<a href="https://colab.research.google.com/github/LP-D/claude/blob/main/notebooks/VIX_ML3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# VIX ML3 — Base Massive pour Stacking
## 9465 modèles ML × 6 horizons × 4 régimes + Architectures DL intégrées
## Features : Kalman · HMM · EGARCH · Heston · VRP · Hawkes + 465 features marché


In [1]:
import subprocess, sys
pkgs = ['xgboost','lightgbm','yfinance','pandas_datareader','arch',
        'pykalman','hmmlearn','shap','xlsxwriter','imbalanced-learn',
        'statsmodels','seaborn','torch']
subprocess.run([sys.executable,'-m','pip','install','-q']+pkgs, check=False)
print("Installation OK")


Installation OK


In [2]:
import os,time,json,warnings,random
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import yfinance as yf
import pandas_datareader.data as web
import shap
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import RobustScaler
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, ExtraTreesClassifier
from sklearn.linear_model import LogisticRegression, RidgeClassifier
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import f1_score, accuracy_score
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from imblearn.over_sampling import SMOTE, BorderlineSMOTE, ADASYN
from imblearn.combine import SMOTETomek, SMOTEENN
from arch import arch_model
from pykalman import KalmanFilter
from hmmlearn import hmm as hmmlib
import statsmodels.api as sm

SEED=42; random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")


Device: cpu


In [3]:
CONFIG = {
    'seed':42,'start_date':'2000-01-01','flat_thr':0.003,
    'horizons':[1,2,3,5,7,10],
    'min_f1_dir':0.50,'max_feat_overlap':0.75,'max_pred_corr':0.85,
    'meta_n_estimators':500,'meta_max_depth':5,'meta_lr':0.02,
    'n_folds_oof':5,'dl_lookback':21,'dl_epochs':50,'dl_batch':64,
    'dl_lr':3e-4,'dl_dropout':0.3,
}
TARGET_COL='VIX_Amplitude_Class'

YF_TICKERS="""
^GSPC ^IXIC ^VIX ^VXN ^OVX ^GVZ ^EVZ ^VVIX
^FTSE ^N225 ^HSI ^GDAXI ^STOXX50E
SPY QQQ TLT GLD USO HYG LQD EEM EFA VNQ
AAPL AMZN MSFT NVDA INTC QCOM XOM WMT MCD SBUX
MS COF BLK SCHW CLX CPB LMT NOC GD HON
CCI PSA EQIX NEE TXN PAYX LUV CMCSA
XLK XLF XLE XLV XLU XLB XLI XLY
AMD AORD ASX AVB AXP BDX BTI BA
""".split()

FRED_SERIES={'NFCI':'NFCI','STLFSI':'STLFSI4','T10Y2Y':'T10Y2Y','EFFR':'EFFR'}

DL_REFS={1:{'F1_dir':0.5528,'F1_UP_FORT':0.296,'F1_DOWN_FORT':0.330},
          3:{'F1_dir':0.6188,'F1_UP_FORT':0.481,'F1_DOWN_FORT':0.301},
          5:{'F1_dir':0.6225,'F1_UP_FORT':0.588,'F1_DOWN_FORT':0.089},
          7:{'F1_dir':0.6128,'F1_UP_FORT':0.549,'F1_DOWN_FORT':0.226}}

ML_REFS={'LogReg h=5j':{'F1_dir':0.634,'F1_UP_FORT':0.406,'F1_DOWN_FORT':0.575},
          'RF h=5j':{'F1_dir':0.620,'F1_UP_FORT':0.412,'F1_DOWN_FORT':0.462},
          'Stacking V1':{'F1_dir':0.596}}

print("Config OK")


Config OK


In [4]:
ALL_RUNS=json.loads('[{"model_id": "v2_h1_CALM_XGBoost_N5", "algo": "XGBoost", "regime": "CALM", "horizon": 1, "n_features": 5, "F1_dir": 0.5787, "F1_UP_FORT": 0.2955, "F1_DOWN_FORT": 0.3404, "train_start": "2001-08-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VIX_Price_zscore_60d__ret5x__LLY_ret_1d", "hmm_p_stress__minus__spx_vol_5d", "XOM_ret_1d", "EWQ_France_ret_1d__minus__BDX_Becton_Dickinson_ret_1d", "MS_MorganStanley_ret_5d"], "is_new": false}, {"model_id": "v2_h1_CALM_XGBoost_N6", "algo": "XGBoost", "regime": "CALM", "horizon": 1, "n_features": 6, "F1_dir": 0.5644, "F1_UP_FORT": 0.2353, "F1_DOWN_FORT": 0.4086, "train_start": "2001-08-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VIX_Price_zscore_60d__ret5x__LLY_ret_1d", "hmm_p_stress__minus__spx_vol_5d", "XOM_ret_1d", "EWQ_France_ret_1d__minus__BDX_Becton_Dickinson_ret_1d", "MS_MorganStanley_ret_5d", "VIX_Price_zscore_60d__div__VRP"], "is_new": false}, {"model_id": "v2_h1_CALM_XGBoost_N7", "algo": "XGBoost", "regime": "CALM", "horizon": 1, "n_features": 7, "F1_dir": 0.5778, "F1_UP_FORT": 0.2, "F1_DOWN_FORT": 0.3596, "train_start": "2001-08-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VIX_Price_zscore_60d__ret5x__LLY_ret_1d", "hmm_p_stress__minus__spx_vol_5d", "XOM_ret_1d", "EWQ_France_ret_1d__minus__BDX_Becton_Dickinson_ret_1d", "MS_MorganStanley_ret_5d", "VIX_Price_zscore_60d__div__VRP", "MSTR_Bitcoin3_ret_5d"], "is_new": false}, {"model_id": "v2_h1_CALM_XGBoost_N8", "algo": "XGBoost", "regime": "CALM", "horizon": 1, "n_features": 8, "F1_dir": 0.5543, "F1_UP_FORT": 0.1333, "F1_DOWN_FORT": 0.3529, "train_start": "2001-08-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VIX_Price_zscore_60d__ret5x__LLY_ret_1d", "hmm_p_stress__minus__spx_vol_5d", "XOM_ret_1d", "EWQ_France_ret_1d__minus__BDX_Becton_Dickinson_ret_1d", "MS_MorganStanley_ret_5d", "VIX_Price_zscore_60d__div__VRP", "MSTR_Bitcoin3_ret_5d", "EOG_EOGResources_ret_5d"], "is_new": false}, {"model_id": "v2_h1_CALM_XGBoost_N9", "algo": "XGBoost", "regime": "CALM", "horizon": 1, "n_features": 9, "F1_dir": 0.6027, "F1_UP_FORT": 0.1429, "F1_DOWN_FORT": 0.3301, "train_start": "2001-08-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VIX_Price_zscore_60d__ret5x__LLY_ret_1d", "hmm_p_stress__minus__spx_vol_5d", "XOM_ret_1d", "EWQ_France_ret_1d__minus__BDX_Becton_Dickinson_ret_1d", "MS_MorganStanley_ret_5d", "VIX_Price_zscore_60d__div__VRP", "MSTR_Bitcoin3_ret_5d", "EOG_EOGResources_ret_5d", "VIX_Price_zscore_60d__div__spx_vol_5d"], "is_new": false}, {"model_id": "v2_h1_CALM_XGBoost_N10", "algo": "XGBoost", "regime": "CALM", "horizon": 1, "n_features": 10, "F1_dir": 0.5926, "F1_UP_FORT": 0.1975, "F1_DOWN_FORT": 0.3368, "train_start": "2001-08-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VIX_Price_zscore_60d__ret5x__LLY_ret_1d", "hmm_p_stress__minus__spx_vol_5d", "XOM_ret_1d", "EWQ_France_ret_1d__minus__BDX_Becton_Dickinson_ret_1d", "MS_MorganStanley_ret_5d", "VIX_Price_zscore_60d__div__VRP", "MSTR_Bitcoin3_ret_5d", "EOG_EOGResources_ret_5d", "VIX_Price_zscore_60d__div__spx_vol_5d", "VIX_Price_zscore_60d__div__vix_vol_of_vol_5d"], "is_new": false}, {"model_id": "v2_h1_CALM_XGBoost_N11", "algo": "XGBoost", "regime": "CALM", "horizon": 1, "n_features": 11, "F1_dir": 0.5926, "F1_UP_FORT": 0.2045, "F1_DOWN_FORT": 0.25, "train_start": "2001-08-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VIX_Price_zscore_60d__ret5x__LLY_ret_1d", "hmm_p_stress__minus__spx_vol_5d", "XOM_ret_1d", "EWQ_France_ret_1d__minus__BDX_Becton_Dickinson_ret_1d", "MS_MorganStanley_ret_5d", "VIX_Price_zscore_60d__div__VRP", "MSTR_Bitcoin3_ret_5d", "EOG_EOGResources_ret_5d", "VIX_Price_zscore_60d__div__spx_vol_5d", "VIX_Price_zscore_60d__div__vix_vol_of_vol_5d", "ITT_ITTInc_ret_5d"], "is_new": false}, {"model_id": "v2_h1_CALM_XGBoost_N12", "algo": "XGBoost", "regime": "CALM", "horizon": 1, "n_features": 12, "F1_dir": 0.618, "F1_UP_FORT": 0.137, "F1_DOWN_FORT": 0.2828, "train_start": "2001-08-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VIX_Price_zscore_60d__ret5x__LLY_ret_1d", "hmm_p_stress__minus__spx_vol_5d", "XOM_ret_1d", "EWQ_France_ret_1d__minus__BDX_Becton_Dickinson_ret_1d", "MS_MorganStanley_ret_5d", "VIX_Price_zscore_60d__div__VRP", "MSTR_Bitcoin3_ret_5d", "EOG_EOGResources_ret_5d", "VIX_Price_zscore_60d__div__spx_vol_5d", "VIX_Price_zscore_60d__div__vix_vol_of_vol_5d", "ITT_ITTInc_ret_5d", "NVDA_vol_20d"], "is_new": false}, {"model_id": "v2_h1_CALM_XGBoost_N13", "algo": "XGBoost", "regime": "CALM", "horizon": 1, "n_features": 13, "F1_dir": 0.5741, "F1_UP_FORT": 0.0822, "F1_DOWN_FORT": 0.2885, "train_start": "2001-08-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VIX_Price_zscore_60d__ret5x__LLY_ret_1d", "hmm_p_stress__minus__spx_vol_5d", "XOM_ret_1d", "EWQ_France_ret_1d__minus__BDX_Becton_Dickinson_ret_1d", "MS_MorganStanley_ret_5d", "VIX_Price_zscore_60d__div__VRP", "MSTR_Bitcoin3_ret_5d", "EOG_EOGResources_ret_5d", "VIX_Price_zscore_60d__div__spx_vol_5d", "VIX_Price_zscore_60d__div__vix_vol_of_vol_5d", "ITT_ITTInc_ret_5d", "NVDA_vol_20d", "hmm_p_stress__zrel__spx_vol_5d"], "is_new": false}, {"model_id": "v2_h1_CALM_XGBoost_N14", "algo": "XGBoost", "regime": "CALM", "horizon": 1, "n_features": 14, "F1_dir": 0.5314, "F1_UP_FORT": 0.1558, "F1_DOWN_FORT": 0.2941, "train_start": "2001-08-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VIX_Price_zscore_60d__ret5x__LLY_ret_1d", "hmm_p_stress__minus__spx_vol_5d", "XOM_ret_1d", "EWQ_France_ret_1d__minus__BDX_Becton_Dickinson_ret_1d", "MS_MorganStanley_ret_5d", "VIX_Price_zscore_60d__div__VRP", "MSTR_Bitcoin3_ret_5d", "EOG_EOGResources_ret_5d", "VIX_Price_zscore_60d__div__spx_vol_5d", "VIX_Price_zscore_60d__div__vix_vol_of_vol_5d", "ITT_ITTInc_ret_5d", "NVDA_vol_20d", "hmm_p_stress__zrel__spx_vol_5d", "MS_MorganStanley_zscore_60d"], "is_new": false}, {"model_id": "v2_h1_CALM_XGBoost_N15", "algo": "XGBoost", "regime": "CALM", "horizon": 1, "n_features": 15, "F1_dir": 0.5507, "F1_UP_FORT": 0.1333, "F1_DOWN_FORT": 0.2941, "train_start": "2001-08-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VIX_Price_zscore_60d__ret5x__LLY_ret_1d", "hmm_p_stress__minus__spx_vol_5d", "XOM_ret_1d", "EWQ_France_ret_1d__minus__BDX_Becton_Dickinson_ret_1d", "MS_MorganStanley_ret_5d", "VIX_Price_zscore_60d__div__VRP", "MSTR_Bitcoin3_ret_5d", "EOG_EOGResources_ret_5d", "VIX_Price_zscore_60d__div__spx_vol_5d", "VIX_Price_zscore_60d__div__vix_vol_of_vol_5d", "ITT_ITTInc_ret_5d", "NVDA_vol_20d", "hmm_p_stress__zrel__spx_vol_5d", "MS_MorganStanley_zscore_60d", "STLFSI4_ret_1d__ret5x__VRP"], "is_new": false}, {"model_id": "v2_h1_CALM_XGBoost_N16", "algo": "XGBoost", "regime": "CALM", "horizon": 1, "n_features": 16, "F1_dir": 0.569, "F1_UP_FORT": 0.1818, "F1_DOWN_FORT": 0.3232, "train_start": "2001-08-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VIX_Price_zscore_60d__ret5x__LLY_ret_1d", "hmm_p_stress__minus__spx_vol_5d", "XOM_ret_1d", "EWQ_France_ret_1d__minus__BDX_Becton_Dickinson_ret_1d", "MS_MorganStanley_ret_5d", "VIX_Price_zscore_60d__div__VRP", "MSTR_Bitcoin3_ret_5d", "EOG_EOGResources_ret_5d", "VIX_Price_zscore_60d__div__spx_vol_5d", "VIX_Price_zscore_60d__div__vix_vol_of_vol_5d", "ITT_ITTInc_ret_5d", "NVDA_vol_20d", "hmm_p_stress__zrel__spx_vol_5d", "MS_MorganStanley_zscore_60d", "STLFSI4_ret_1d__ret5x__VRP", "AMGN_Amgen_ret_1d"], "is_new": false}, {"model_id": "v2_h1_CALM_XGBoost_N17", "algo": "XGBoost", "regime": "CALM", "horizon": 1, "n_features": 17, "F1_dir": 0.5553, "F1_UP_FORT": 0.137, "F1_DOWN_FORT": 0.3301, "train_start": "2001-08-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VIX_Price_zscore_60d__ret5x__LLY_ret_1d", "hmm_p_stress__minus__spx_vol_5d", "XOM_ret_1d", "EWQ_France_ret_1d__minus__BDX_Becton_Dickinson_ret_1d", "MS_MorganStanley_ret_5d", "VIX_Price_zscore_60d__div__VRP", "MSTR_Bitcoin3_ret_5d", "EOG_EOGResources_ret_5d", "VIX_Price_zscore_60d__div__spx_vol_5d", "VIX_Price_zscore_60d__div__vix_vol_of_vol_5d", "ITT_ITTInc_ret_5d", "NVDA_vol_20d", "hmm_p_stress__zrel__spx_vol_5d", "MS_MorganStanley_zscore_60d", "STLFSI4_ret_1d__ret5x__VRP", "AMGN_Amgen_ret_1d", "NFCI_ret_5d"], "is_new": false}, {"model_id": "v2_h1_CALM_XGBoost_N18", "algo": "XGBoost", "regime": "CALM", "horizon": 1, "n_features": 18, "F1_dir": 0.5699, "F1_UP_FORT": 0.1818, "F1_DOWN_FORT": 0.3232, "train_start": "2001-08-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VIX_Price_zscore_60d__ret5x__LLY_ret_1d", "hmm_p_stress__minus__spx_vol_5d", "XOM_ret_1d", "EWQ_France_ret_1d__minus__BDX_Becton_Dickinson_ret_1d", "MS_MorganStanley_ret_5d", "VIX_Price_zscore_60d__div__VRP", "MSTR_Bitcoin3_ret_5d", "EOG_EOGResources_ret_5d", "VIX_Price_zscore_60d__div__spx_vol_5d", "VIX_Price_zscore_60d__div__vix_vol_of_vol_5d", "ITT_ITTInc_ret_5d", "NVDA_vol_20d", "hmm_p_stress__zrel__spx_vol_5d", "MS_MorganStanley_zscore_60d", "STLFSI4_ret_1d__ret5x__VRP", "AMGN_Amgen_ret_1d", "NFCI_ret_5d", "STLFSI4_ret_1d__minus__spx_vol_5d"], "is_new": false}, {"model_id": "v2_h1_CALM_XGBoost_N19", "algo": "XGBoost", "regime": "CALM", "horizon": 1, "n_features": 19, "F1_dir": 0.5409, "F1_UP_FORT": 0.1389, "F1_DOWN_FORT": 0.3137, "train_start": "2001-08-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VIX_Price_zscore_60d__ret5x__LLY_ret_1d", "hmm_p_stress__minus__spx_vol_5d", "XOM_ret_1d", "EWQ_France_ret_1d__minus__BDX_Becton_Dickinson_ret_1d", "MS_MorganStanley_ret_5d", "VIX_Price_zscore_60d__div__VRP", "MSTR_Bitcoin3_ret_5d", "EOG_EOGResources_ret_5d", "VIX_Price_zscore_60d__div__spx_vol_5d", "VIX_Price_zscore_60d__div__vix_vol_of_vol_5d", "ITT_ITTInc_ret_5d", "NVDA_vol_20d", "hmm_p_stress__zrel__spx_vol_5d", "MS_MorganStanley_zscore_60d", "STLFSI4_ret_1d__ret5x__VRP", "AMGN_Amgen_ret_1d", "NFCI_ret_5d", "STLFSI4_ret_1d__minus__spx_vol_5d", "STLFSI4_ret_1d__macross__BDX_Becton_Dickinson_ret_1d"], "is_new": false}, {"model_id": "v2_h1_CALM_LightGBM_N5", "algo": "LightGBM", "regime": "CALM", "horizon": 1, "n_features": 5, "F1_dir": 0.5589, "F1_UP_FORT": 0.25, "F1_DOWN_FORT": 0.303, "train_start": "2001-08-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VIX_Price_zscore_60d__ret5x__LLY_ret_1d", "hmm_p_stress__minus__spx_vol_5d", "XOM_ret_1d", "EWQ_France_ret_1d__minus__BDX_Becton_Dickinson_ret_1d", "MS_MorganStanley_ret_5d"], "is_new": false}, {"model_id": "v2_h1_CALM_LightGBM_N6", "algo": "LightGBM", "regime": "CALM", "horizon": 1, "n_features": 6, "F1_dir": 0.5765, "F1_UP_FORT": 0.2222, "F1_DOWN_FORT": 0.3505, "train_start": "2001-08-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VIX_Price_zscore_60d__ret5x__LLY_ret_1d", "hmm_p_stress__minus__spx_vol_5d", "XOM_ret_1d", "EWQ_France_ret_1d__minus__BDX_Becton_Dickinson_ret_1d", "MS_MorganStanley_ret_5d", "VIX_Price_zscore_60d__div__VRP"], "is_new": false}, {"model_id": "v2_h1_CALM_LightGBM_N7", "algo": "LightGBM", "regime": "CALM", "horizon": 1, "n_features": 7, "F1_dir": 0.6277, "F1_UP_FORT": 0.1519, "F1_DOWN_FORT": 0.34, "train_start": "2001-08-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VIX_Price_zscore_60d__ret5x__LLY_ret_1d", "hmm_p_stress__minus__spx_vol_5d", "XOM_ret_1d", "EWQ_France_ret_1d__minus__BDX_Becton_Dickinson_ret_1d", "MS_MorganStanley_ret_5d", "VIX_Price_zscore_60d__div__VRP", "MSTR_Bitcoin3_ret_5d"], "is_new": false}, {"model_id": "v2_h1_CALM_LightGBM_N8", "algo": "LightGBM", "regime": "CALM", "horizon": 1, "n_features": 8, "F1_dir": 0.6027, "F1_UP_FORT": 0.1839, "F1_DOWN_FORT": 0.3269, "train_start": "2001-08-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VIX_Price_zscore_60d__ret5x__LLY_ret_1d", "hmm_p_stress__minus__spx_vol_5d", "XOM_ret_1d", "EWQ_France_ret_1d__minus__BDX_Becton_Dickinson_ret_1d", "MS_MorganStanley_ret_5d", "VIX_Price_zscore_60d__div__VRP", "MSTR_Bitcoin3_ret_5d", "EOG_EOGResources_ret_5d"], "is_new": false}, {"model_id": "v2_h1_CALM_LightGBM_N9", "algo": "LightGBM", "regime": "CALM", "horizon": 1, "n_features": 9, "F1_dir": 0.596, "F1_UP_FORT": 0.2093, "F1_DOWN_FORT": 0.3654, "train_start": "2001-08-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VIX_Price_zscore_60d__ret5x__LLY_ret_1d", "hmm_p_stress__minus__spx_vol_5d", "XOM_ret_1d", "EWQ_France_ret_1d__minus__BDX_Becton_Dickinson_ret_1d", "MS_MorganStanley_ret_5d", "VIX_Price_zscore_60d__div__VRP", "MSTR_Bitcoin3_ret_5d", "EOG_EOGResources_ret_5d", "VIX_Price_zscore_60d__div__spx_vol_5d"], "is_new": false}, {"model_id": "v2_h1_CALM_LightGBM_N10", "algo": "LightGBM", "regime": "CALM", "horizon": 1, "n_features": 10, "F1_dir": 0.5772, "F1_UP_FORT": 0.1687, "F1_DOWN_FORT": 0.2737, "train_start": "2001-08-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VIX_Price_zscore_60d__ret5x__LLY_ret_1d", "hmm_p_stress__minus__spx_vol_5d", "XOM_ret_1d", "EWQ_France_ret_1d__minus__BDX_Becton_Dickinson_ret_1d", "MS_MorganStanley_ret_5d", "VIX_Price_zscore_60d__div__VRP", "MSTR_Bitcoin3_ret_5d", "EOG_EOGResources_ret_5d", "VIX_Price_zscore_60d__div__spx_vol_5d", "VIX_Price_zscore_60d__div__vix_vol_of_vol_5d"], "is_new": false}, {"model_id": "v2_h1_CALM_LightGBM_N11", "algo": "LightGBM", "regime": "CALM", "horizon": 1, "n_features": 11, "F1_dir": 0.5797, "F1_UP_FORT": 0.1975, "F1_DOWN_FORT": 0.25, "train_start": "2001-08-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VIX_Price_zscore_60d__ret5x__LLY_ret_1d", "hmm_p_stress__minus__spx_vol_5d", "XOM_ret_1d", "EWQ_France_ret_1d__minus__BDX_Becton_Dickinson_ret_1d", "MS_MorganStanley_ret_5d", "VIX_Price_zscore_60d__div__VRP", "MSTR_Bitcoin3_ret_5d", "EOG_EOGResources_ret_5d", "VIX_Price_zscore_60d__div__spx_vol_5d", "VIX_Price_zscore_60d__div__vix_vol_of_vol_5d", "ITT_ITTInc_ret_5d"], "is_new": false}, {"model_id": "v2_h1_CALM_LightGBM_N12", "algo": "LightGBM", "regime": "CALM", "horizon": 1, "n_features": 12, "F1_dir": 0.565, "F1_UP_FORT": 0.0811, "F1_DOWN_FORT": 0.2887, "train_start": "2001-08-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VIX_Price_zscore_60d__ret5x__LLY_ret_1d", "hmm_p_stress__minus__spx_vol_5d", "XOM_ret_1d", "EWQ_France_ret_1d__minus__BDX_Becton_Dickinson_ret_1d", "MS_MorganStanley_ret_5d", "VIX_Price_zscore_60d__div__VRP", "MSTR_Bitcoin3_ret_5d", "EOG_EOGResources_ret_5d", "VIX_Price_zscore_60d__div__spx_vol_5d", "VIX_Price_zscore_60d__div__vix_vol_of_vol_5d", "ITT_ITTInc_ret_5d", "NVDA_vol_20d"], "is_new": false}, {"model_id": "v2_h1_CALM_LightGBM_N13", "algo": "LightGBM", "regime": "CALM", "horizon": 1, "n_features": 13, "F1_dir": 0.5651, "F1_UP_FORT": 0.1412, "F1_DOWN_FORT": 0.2574, "train_start": "2001-08-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VIX_Price_zscore_60d__ret5x__LLY_ret_1d", "hmm_p_stress__minus__spx_vol_5d", "XOM_ret_1d", "EWQ_France_ret_1d__minus__BDX_Becton_Dickinson_ret_1d", "MS_MorganStanley_ret_5d", "VIX_Price_zscore_60d__div__VRP", "MSTR_Bitcoin3_ret_5d", "EOG_EOGResources_ret_5d", "VIX_Price_zscore_60d__div__spx_vol_5d", "VIX_Price_zscore_60d__div__vix_vol_of_vol_5d", "ITT_ITTInc_ret_5d", "NVDA_vol_20d", "hmm_p_stress__zrel__spx_vol_5d"], "is_new": false}, {"model_id": "v2_h1_CALM_LightGBM_N14", "algo": "LightGBM", "regime": "CALM", "horizon": 1, "n_features": 14, "F1_dir": 0.5748, "F1_UP_FORT": 0.2105, "F1_DOWN_FORT": 0.297, "train_start": "2001-08-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VIX_Price_zscore_60d__ret5x__LLY_ret_1d", "hmm_p_stress__minus__spx_vol_5d", "XOM_ret_1d", "EWQ_France_ret_1d__minus__BDX_Becton_Dickinson_ret_1d", "MS_MorganStanley_ret_5d", "VIX_Price_zscore_60d__div__VRP", "MSTR_Bitcoin3_ret_5d", "EOG_EOGResources_ret_5d", "VIX_Price_zscore_60d__div__spx_vol_5d", "VIX_Price_zscore_60d__div__vix_vol_of_vol_5d", "ITT_ITTInc_ret_5d", "NVDA_vol_20d", "hmm_p_stress__zrel__spx_vol_5d", "MS_MorganStanley_zscore_60d"], "is_new": false}, {"model_id": "v2_h1_CALM_LightGBM_N15", "algo": "LightGBM", "regime": "CALM", "horizon": 1, "n_features": 15, "F1_dir": 0.5584, "F1_UP_FORT": 0.125, "F1_DOWN_FORT": 0.2947, "train_start": "2001-08-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VIX_Price_zscore_60d__ret5x__LLY_ret_1d", "hmm_p_stress__minus__spx_vol_5d", "XOM_ret_1d", "EWQ_France_ret_1d__minus__BDX_Becton_Dickinson_ret_1d", "MS_MorganStanley_ret_5d", "VIX_Price_zscore_60d__div__VRP", "MSTR_Bitcoin3_ret_5d", "EOG_EOGResources_ret_5d", "VIX_Price_zscore_60d__div__spx_vol_5d", "VIX_Price_zscore_60d__div__vix_vol_of_vol_5d", "ITT_ITTInc_ret_5d", "NVDA_vol_20d", "hmm_p_stress__zrel__spx_vol_5d", "MS_MorganStanley_zscore_60d", "STLFSI4_ret_1d__ret5x__VRP"], "is_new": false}, {"model_id": "v2_h1_CALM_LightGBM_N16", "algo": "LightGBM", "regime": "CALM", "horizon": 1, "n_features": 16, "F1_dir": 0.5307, "F1_UP_FORT": 0.1299, "F1_DOWN_FORT": 0.2198, "train_start": "2001-08-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VIX_Price_zscore_60d__ret5x__LLY_ret_1d", "hmm_p_stress__minus__spx_vol_5d", "XOM_ret_1d", "EWQ_France_ret_1d__minus__BDX_Becton_Dickinson_ret_1d", "MS_MorganStanley_ret_5d", "VIX_Price_zscore_60d__div__VRP", "MSTR_Bitcoin3_ret_5d", "EOG_EOGResources_ret_5d", "VIX_Price_zscore_60d__div__spx_vol_5d", "VIX_Price_zscore_60d__div__vix_vol_of_vol_5d", "ITT_ITTInc_ret_5d", "NVDA_vol_20d", "hmm_p_stress__zrel__spx_vol_5d", "MS_MorganStanley_zscore_60d", "STLFSI4_ret_1d__ret5x__VRP", "AMGN_Amgen_ret_1d"], "is_new": false}, {"model_id": "v2_h1_CALM_LightGBM_N17", "algo": "LightGBM", "regime": "CALM", "horizon": 1, "n_features": 17, "F1_dir": 0.5492, "F1_UP_FORT": 0.1408, "F1_DOWN_FORT": 0.2558, "train_start": "2001-08-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VIX_Price_zscore_60d__ret5x__LLY_ret_1d", "hmm_p_stress__minus__spx_vol_5d", "XOM_ret_1d", "EWQ_France_ret_1d__minus__BDX_Becton_Dickinson_ret_1d", "MS_MorganStanley_ret_5d", "VIX_Price_zscore_60d__div__VRP", "MSTR_Bitcoin3_ret_5d", "EOG_EOGResources_ret_5d", "VIX_Price_zscore_60d__div__spx_vol_5d", "VIX_Price_zscore_60d__div__vix_vol_of_vol_5d", "ITT_ITTInc_ret_5d", "NVDA_vol_20d", "hmm_p_stress__zrel__spx_vol_5d", "MS_MorganStanley_zscore_60d", "STLFSI4_ret_1d__ret5x__VRP", "AMGN_Amgen_ret_1d", "NFCI_ret_5d"], "is_new": false}, {"model_id": "v2_h1_CALM_LightGBM_N18", "algo": "LightGBM", "regime": "CALM", "horizon": 1, "n_features": 18, "F1_dir": 0.6135, "F1_UP_FORT": 0.2133, "F1_DOWN_FORT": 0.3838, "train_start": "2001-08-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VIX_Price_zscore_60d__ret5x__LLY_ret_1d", "hmm_p_stress__minus__spx_vol_5d", "XOM_ret_1d", "EWQ_France_ret_1d__minus__BDX_Becton_Dickinson_ret_1d", "MS_MorganStanley_ret_5d", "VIX_Price_zscore_60d__div__VRP", "MSTR_Bitcoin3_ret_5d", "EOG_EOGResources_ret_5d", "VIX_Price_zscore_60d__div__spx_vol_5d", "VIX_Price_zscore_60d__div__vix_vol_of_vol_5d", "ITT_ITTInc_ret_5d", "NVDA_vol_20d", "hmm_p_stress__zrel__spx_vol_5d", "MS_MorganStanley_zscore_60d", "STLFSI4_ret_1d__ret5x__VRP", "AMGN_Amgen_ret_1d", "NFCI_ret_5d", "STLFSI4_ret_1d__minus__spx_vol_5d"], "is_new": false}, {"model_id": "v2_h1_CALM_LightGBM_N19", "algo": "LightGBM", "regime": "CALM", "horizon": 1, "n_features": 19, "F1_dir": 0.5744, "F1_UP_FORT": 0.1667, "F1_DOWN_FORT": 0.2979, "train_start": "2001-08-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VIX_Price_zscore_60d__ret5x__LLY_ret_1d", "hmm_p_stress__minus__spx_vol_5d", "XOM_ret_1d", "EWQ_France_ret_1d__minus__BDX_Becton_Dickinson_ret_1d", "MS_MorganStanley_ret_5d", "VIX_Price_zscore_60d__div__VRP", "MSTR_Bitcoin3_ret_5d", "EOG_EOGResources_ret_5d", "VIX_Price_zscore_60d__div__spx_vol_5d", "VIX_Price_zscore_60d__div__vix_vol_of_vol_5d", "ITT_ITTInc_ret_5d", "NVDA_vol_20d", "hmm_p_stress__zrel__spx_vol_5d", "MS_MorganStanley_zscore_60d", "STLFSI4_ret_1d__ret5x__VRP", "AMGN_Amgen_ret_1d", "NFCI_ret_5d", "STLFSI4_ret_1d__minus__spx_vol_5d", "STLFSI4_ret_1d__macross__BDX_Becton_Dickinson_ret_1d"], "is_new": false}, {"model_id": "v2_h1_CALM_GradientBoosting_N5", "algo": "GradientBoosting", "regime": "CALM", "horizon": 1, "n_features": 5, "F1_dir": 0.5694, "F1_UP_FORT": 0.2045, "F1_DOWN_FORT": 0.3656, "train_start": "2001-08-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VIX_Price_zscore_60d__ret5x__LLY_ret_1d", "hmm_p_stress__minus__spx_vol_5d", "XOM_ret_1d", "EWQ_France_ret_1d__minus__BDX_Becton_Dickinson_ret_1d", "MS_MorganStanley_ret_5d"], "is_new": false}, {"model_id": "v2_h1_CALM_GradientBoosting_N6", "algo": "GradientBoosting", "regime": "CALM", "horizon": 1, "n_features": 6, "F1_dir": 0.5744, "F1_UP_FORT": 0.2247, "F1_DOWN_FORT": 0.3878, "train_start": "2001-08-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VIX_Price_zscore_60d__ret5x__LLY_ret_1d", "hmm_p_stress__minus__spx_vol_5d", "XOM_ret_1d", "EWQ_France_ret_1d__minus__BDX_Becton_Dickinson_ret_1d", "MS_MorganStanley_ret_5d", "VIX_Price_zscore_60d__div__VRP"], "is_new": false}, {"model_id": "v2_h1_CALM_GradientBoosting_N7", "algo": "GradientBoosting", "regime": "CALM", "horizon": 1, "n_features": 7, "F1_dir": 0.5892, "F1_UP_FORT": 0.1707, "F1_DOWN_FORT": 0.3956, "train_start": "2001-08-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VIX_Price_zscore_60d__ret5x__LLY_ret_1d", "hmm_p_stress__minus__spx_vol_5d", "XOM_ret_1d", "EWQ_France_ret_1d__minus__BDX_Becton_Dickinson_ret_1d", "MS_MorganStanley_ret_5d", "VIX_Price_zscore_60d__div__VRP", "MSTR_Bitcoin3_ret_5d"], "is_new": false}, {"model_id": "v2_h1_CALM_GradientBoosting_N8", "algo": "GradientBoosting", "regime": "CALM", "horizon": 1, "n_features": 8, "F1_dir": 0.6069, "F1_UP_FORT": 0.1266, "F1_DOWN_FORT": 0.375, "train_start": "2001-08-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VIX_Price_zscore_60d__ret5x__LLY_ret_1d", "hmm_p_stress__minus__spx_vol_5d", "XOM_ret_1d", "EWQ_France_ret_1d__minus__BDX_Becton_Dickinson_ret_1d", "MS_MorganStanley_ret_5d", "VIX_Price_zscore_60d__div__VRP", "MSTR_Bitcoin3_ret_5d", "EOG_EOGResources_ret_5d"], "is_new": false}, {"model_id": "v2_h1_CALM_GradientBoosting_N9", "algo": "GradientBoosting", "regime": "CALM", "horizon": 1, "n_features": 9, "F1_dir": 0.6018, "F1_UP_FORT": 0.1882, "F1_DOWN_FORT": 0.4043, "train_start": "2001-08-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VIX_Price_zscore_60d__ret5x__LLY_ret_1d", "hmm_p_stress__minus__spx_vol_5d", "XOM_ret_1d", "EWQ_France_ret_1d__minus__BDX_Becton_Dickinson_ret_1d", "MS_MorganStanley_ret_5d", "VIX_Price_zscore_60d__div__VRP", "MSTR_Bitcoin3_ret_5d", "EOG_EOGResources_ret_5d", "VIX_Price_zscore_60d__div__spx_vol_5d"], "is_new": false}, {"model_id": "v2_h1_CALM_GradientBoosting_N10", "algo": "GradientBoosting", "regime": "CALM", "horizon": 1, "n_features": 10, "F1_dir": 0.5602, "F1_UP_FORT": 0.1905, "F1_DOWN_FORT": 0.3261, "train_start": "2001-08-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VIX_Price_zscore_60d__ret5x__LLY_ret_1d", "hmm_p_stress__minus__spx_vol_5d", "XOM_ret_1d", "EWQ_France_ret_1d__minus__BDX_Becton_Dickinson_ret_1d", "MS_MorganStanley_ret_5d", "VIX_Price_zscore_60d__div__VRP", "MSTR_Bitcoin3_ret_5d", "EOG_EOGResources_ret_5d", "VIX_Price_zscore_60d__div__spx_vol_5d", "VIX_Price_zscore_60d__div__vix_vol_of_vol_5d"], "is_new": false}, {"model_id": "v2_h1_CALM_GradientBoosting_N11", "algo": "GradientBoosting", "regime": "CALM", "horizon": 1, "n_features": 11, "F1_dir": 0.5744, "F1_UP_FORT": 0.1667, "F1_DOWN_FORT": 0.3434, "train_start": "2001-08-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VIX_Price_zscore_60d__ret5x__LLY_ret_1d", "hmm_p_stress__minus__spx_vol_5d", "XOM_ret_1d", "EWQ_France_ret_1d__minus__BDX_Becton_Dickinson_ret_1d", "MS_MorganStanley_ret_5d", "VIX_Price_zscore_60d__div__VRP", "MSTR_Bitcoin3_ret_5d", "EOG_EOGResources_ret_5d", "VIX_Price_zscore_60d__div__spx_vol_5d", "VIX_Price_zscore_60d__div__vix_vol_of_vol_5d", "ITT_ITTInc_ret_5d"], "is_new": false}, {"model_id": "v2_h1_CALM_GradientBoosting_N12", "algo": "GradientBoosting", "regime": "CALM", "horizon": 1, "n_features": 12, "F1_dir": 0.565, "F1_UP_FORT": 0.1081, "F1_DOWN_FORT": 0.2553, "train_start": "2001-08-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VIX_Price_zscore_60d__ret5x__LLY_ret_1d", "hmm_p_stress__minus__spx_vol_5d", "XOM_ret_1d", "EWQ_France_ret_1d__minus__BDX_Becton_Dickinson_ret_1d", "MS_MorganStanley_ret_5d", "VIX_Price_zscore_60d__div__VRP", "MSTR_Bitcoin3_ret_5d", "EOG_EOGResources_ret_5d", "VIX_Price_zscore_60d__div__spx_vol_5d", "VIX_Price_zscore_60d__div__vix_vol_of_vol_5d", "ITT_ITTInc_ret_5d", "NVDA_vol_20d"], "is_new": false}, {"model_id": "v2_h1_CALM_GradientBoosting_N13", "algo": "GradientBoosting", "regime": "CALM", "horizon": 1, "n_features": 13, "F1_dir": 0.5555, "F1_UP_FORT": 0.0556, "F1_DOWN_FORT": 0.2857, "train_start": "2001-08-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VIX_Price_zscore_60d__ret5x__LLY_ret_1d", "hmm_p_stress__minus__spx_vol_5d", "XOM_ret_1d", "EWQ_France_ret_1d__minus__BDX_Becton_Dickinson_ret_1d", "MS_MorganStanley_ret_5d", "VIX_Price_zscore_60d__div__VRP", "MSTR_Bitcoin3_ret_5d", "EOG_EOGResources_ret_5d", "VIX_Price_zscore_60d__div__spx_vol_5d", "VIX_Price_zscore_60d__div__vix_vol_of_vol_5d", "ITT_ITTInc_ret_5d", "NVDA_vol_20d", "hmm_p_stress__zrel__spx_vol_5d"], "is_new": false}, {"model_id": "v2_h1_CALM_GradientBoosting_N14", "algo": "GradientBoosting", "regime": "CALM", "horizon": 1, "n_features": 14, "F1_dir": 0.5603, "F1_UP_FORT": 0.1053, "F1_DOWN_FORT": 0.303, "train_start": "2001-08-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VIX_Price_zscore_60d__ret5x__LLY_ret_1d", "hmm_p_stress__minus__spx_vol_5d", "XOM_ret_1d", "EWQ_France_ret_1d__minus__BDX_Becton_Dickinson_ret_1d", "MS_MorganStanley_ret_5d", "VIX_Price_zscore_60d__div__VRP", "MSTR_Bitcoin3_ret_5d", "EOG_EOGResources_ret_5d", "VIX_Price_zscore_60d__div__spx_vol_5d", "VIX_Price_zscore_60d__div__vix_vol_of_vol_5d", "ITT_ITTInc_ret_5d", "NVDA_vol_20d", "hmm_p_stress__zrel__spx_vol_5d", "MS_MorganStanley_zscore_60d"], "is_new": false}, {"model_id": "v2_h1_CALM_GradientBoosting_N15", "algo": "GradientBoosting", "regime": "CALM", "horizon": 1, "n_features": 15, "F1_dir": 0.541, "F1_UP_FORT": 0.1067, "F1_DOWN_FORT": 0.3429, "train_start": "2001-08-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VIX_Price_zscore_60d__ret5x__LLY_ret_1d", "hmm_p_stress__minus__spx_vol_5d", "XOM_ret_1d", "EWQ_France_ret_1d__minus__BDX_Becton_Dickinson_ret_1d", "MS_MorganStanley_ret_5d", "VIX_Price_zscore_60d__div__VRP", "MSTR_Bitcoin3_ret_5d", "EOG_EOGResources_ret_5d", "VIX_Price_zscore_60d__div__spx_vol_5d", "VIX_Price_zscore_60d__div__vix_vol_of_vol_5d", "ITT_ITTInc_ret_5d", "NVDA_vol_20d", "hmm_p_stress__zrel__spx_vol_5d", "MS_MorganStanley_zscore_60d", "STLFSI4_ret_1d__ret5x__VRP"], "is_new": false}, {"model_id": "v2_h1_CALM_GradientBoosting_N16", "algo": "GradientBoosting", "regime": "CALM", "horizon": 1, "n_features": 16, "F1_dir": 0.5726, "F1_UP_FORT": 0.2025, "F1_DOWN_FORT": 0.3232, "train_start": "2001-08-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VIX_Price_zscore_60d__ret5x__LLY_ret_1d", "hmm_p_stress__minus__spx_vol_5d", "XOM_ret_1d", "EWQ_France_ret_1d__minus__BDX_Becton_Dickinson_ret_1d", "MS_MorganStanley_ret_5d", "VIX_Price_zscore_60d__div__VRP", "MSTR_Bitcoin3_ret_5d", "EOG_EOGResources_ret_5d", "VIX_Price_zscore_60d__div__spx_vol_5d", "VIX_Price_zscore_60d__div__vix_vol_of_vol_5d", "ITT_ITTInc_ret_5d", "NVDA_vol_20d", "hmm_p_stress__zrel__spx_vol_5d", "MS_MorganStanley_zscore_60d", "STLFSI4_ret_1d__ret5x__VRP", "AMGN_Amgen_ret_1d"], "is_new": false}, {"model_id": "v2_h1_CALM_GradientBoosting_N17", "algo": "GradientBoosting", "regime": "CALM", "horizon": 1, "n_features": 17, "F1_dir": 0.5845, "F1_UP_FORT": 0.1143, "F1_DOWN_FORT": 0.2917, "train_start": "2001-08-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VIX_Price_zscore_60d__ret5x__LLY_ret_1d", "hmm_p_stress__minus__spx_vol_5d", "XOM_ret_1d", "EWQ_France_ret_1d__minus__BDX_Becton_Dickinson_ret_1d", "MS_MorganStanley_ret_5d", "VIX_Price_zscore_60d__div__VRP", "MSTR_Bitcoin3_ret_5d", "EOG_EOGResources_ret_5d", "VIX_Price_zscore_60d__div__spx_vol_5d", "VIX_Price_zscore_60d__div__vix_vol_of_vol_5d", "ITT_ITTInc_ret_5d", "NVDA_vol_20d", "hmm_p_stress__zrel__spx_vol_5d", "MS_MorganStanley_zscore_60d", "STLFSI4_ret_1d__ret5x__VRP", "AMGN_Amgen_ret_1d", "NFCI_ret_5d"], "is_new": false}, {"model_id": "v2_h1_CALM_GradientBoosting_N18", "algo": "GradientBoosting", "regime": "CALM", "horizon": 1, "n_features": 18, "F1_dir": 0.5555, "F1_UP_FORT": 0.1867, "F1_DOWN_FORT": 0.3043, "train_start": "2001-08-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VIX_Price_zscore_60d__ret5x__LLY_ret_1d", "hmm_p_stress__minus__spx_vol_5d", "XOM_ret_1d", "EWQ_France_ret_1d__minus__BDX_Becton_Dickinson_ret_1d", "MS_MorganStanley_ret_5d", "VIX_Price_zscore_60d__div__VRP", "MSTR_Bitcoin3_ret_5d", "EOG_EOGResources_ret_5d", "VIX_Price_zscore_60d__div__spx_vol_5d", "VIX_Price_zscore_60d__div__vix_vol_of_vol_5d", "ITT_ITTInc_ret_5d", "NVDA_vol_20d", "hmm_p_stress__zrel__spx_vol_5d", "MS_MorganStanley_zscore_60d", "STLFSI4_ret_1d__ret5x__VRP", "AMGN_Amgen_ret_1d", "NFCI_ret_5d", "STLFSI4_ret_1d__minus__spx_vol_5d"], "is_new": false}, {"model_id": "v2_h1_CALM_GradientBoosting_N19", "algo": "GradientBoosting", "regime": "CALM", "horizon": 1, "n_features": 19, "F1_dir": 0.57, "F1_UP_FORT": 0.1739, "F1_DOWN_FORT": 0.2887, "train_start": "2001-08-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VIX_Price_zscore_60d__ret5x__LLY_ret_1d", "hmm_p_stress__minus__spx_vol_5d", "XOM_ret_1d", "EWQ_France_ret_1d__minus__BDX_Becton_Dickinson_ret_1d", "MS_MorganStanley_ret_5d", "VIX_Price_zscore_60d__div__VRP", "MSTR_Bitcoin3_ret_5d", "EOG_EOGResources_ret_5d", "VIX_Price_zscore_60d__div__spx_vol_5d", "VIX_Price_zscore_60d__div__vix_vol_of_vol_5d", "ITT_ITTInc_ret_5d", "NVDA_vol_20d", "hmm_p_stress__zrel__spx_vol_5d", "MS_MorganStanley_zscore_60d", "STLFSI4_ret_1d__ret5x__VRP", "AMGN_Amgen_ret_1d", "NFCI_ret_5d", "STLFSI4_ret_1d__minus__spx_vol_5d", "STLFSI4_ret_1d__macross__BDX_Becton_Dickinson_ret_1d"], "is_new": false}, {"model_id": "v2_h1_CALM_RandomForest_N5", "algo": "RandomForest", "regime": "CALM", "horizon": 1, "n_features": 5, "F1_dir": 0.56, "F1_UP_FORT": 0.1842, "F1_DOWN_FORT": 0.3784, "train_start": "2001-08-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VIX_Price_zscore_60d__ret5x__LLY_ret_1d", "hmm_p_stress__minus__spx_vol_5d", "XOM_ret_1d", "EWQ_France_ret_1d__minus__BDX_Becton_Dickinson_ret_1d", "MS_MorganStanley_ret_5d"], "is_new": false}, {"model_id": "v2_h1_CALM_RandomForest_N6", "algo": "RandomForest", "regime": "CALM", "horizon": 1, "n_features": 6, "F1_dir": 0.5584, "F1_UP_FORT": 0.169, "F1_DOWN_FORT": 0.3604, "train_start": "2001-08-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VIX_Price_zscore_60d__ret5x__LLY_ret_1d", "hmm_p_stress__minus__spx_vol_5d", "XOM_ret_1d", "EWQ_France_ret_1d__minus__BDX_Becton_Dickinson_ret_1d", "MS_MorganStanley_ret_5d", "VIX_Price_zscore_60d__div__VRP"], "is_new": false}, {"model_id": "v2_h1_CALM_RandomForest_N7", "algo": "RandomForest", "regime": "CALM", "horizon": 1, "n_features": 7, "F1_dir": 0.6183, "F1_UP_FORT": 0.16, "F1_DOWN_FORT": 0.4158, "train_start": "2001-08-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VIX_Price_zscore_60d__ret5x__LLY_ret_1d", "hmm_p_stress__minus__spx_vol_5d", "XOM_ret_1d", "EWQ_France_ret_1d__minus__BDX_Becton_Dickinson_ret_1d", "MS_MorganStanley_ret_5d", "VIX_Price_zscore_60d__div__VRP", "MSTR_Bitcoin3_ret_5d"], "is_new": false}, {"model_id": "v2_h1_CALM_RandomForest_N8", "algo": "RandomForest", "regime": "CALM", "horizon": 1, "n_features": 8, "F1_dir": 0.56, "F1_UP_FORT": 0.1892, "F1_DOWN_FORT": 0.42, "train_start": "2001-08-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VIX_Price_zscore_60d__ret5x__LLY_ret_1d", "hmm_p_stress__minus__spx_vol_5d", "XOM_ret_1d", "EWQ_France_ret_1d__minus__BDX_Becton_Dickinson_ret_1d", "MS_MorganStanley_ret_5d", "VIX_Price_zscore_60d__div__VRP", "MSTR_Bitcoin3_ret_5d", "EOG_EOGResources_ret_5d"], "is_new": false}, {"model_id": "v2_h1_CALM_RandomForest_N9", "algo": "RandomForest", "regime": "CALM", "horizon": 1, "n_features": 9, "F1_dir": 0.5981, "F1_UP_FORT": 0.2222, "F1_DOWN_FORT": 0.3226, "train_start": "2001-08-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VIX_Price_zscore_60d__ret5x__LLY_ret_1d", "hmm_p_stress__minus__spx_vol_5d", "XOM_ret_1d", "EWQ_France_ret_1d__minus__BDX_Becton_Dickinson_ret_1d", "MS_MorganStanley_ret_5d", "VIX_Price_zscore_60d__div__VRP", "MSTR_Bitcoin3_ret_5d", "EOG_EOGResources_ret_5d", "VIX_Price_zscore_60d__div__spx_vol_5d"], "is_new": false}, {"model_id": "v2_h1_CALM_RandomForest_N10", "algo": "RandomForest", "regime": "CALM", "horizon": 1, "n_features": 10, "F1_dir": 0.5409, "F1_UP_FORT": 0.2716, "F1_DOWN_FORT": 0.3125, "train_start": "2001-08-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VIX_Price_zscore_60d__ret5x__LLY_ret_1d", "hmm_p_stress__minus__spx_vol_5d", "XOM_ret_1d", "EWQ_France_ret_1d__minus__BDX_Becton_Dickinson_ret_1d", "MS_MorganStanley_ret_5d", "VIX_Price_zscore_60d__div__VRP", "MSTR_Bitcoin3_ret_5d", "EOG_EOGResources_ret_5d", "VIX_Price_zscore_60d__div__spx_vol_5d", "VIX_Price_zscore_60d__div__vix_vol_of_vol_5d"], "is_new": false}, {"model_id": "v2_h1_CALM_RandomForest_N11", "algo": "RandomForest", "regime": "CALM", "horizon": 1, "n_features": 11, "F1_dir": 0.5409, "F1_UP_FORT": 0.1867, "F1_DOWN_FORT": 0.3148, "train_start": "2001-08-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VIX_Price_zscore_60d__ret5x__LLY_ret_1d", "hmm_p_stress__minus__spx_vol_5d", "XOM_ret_1d", "EWQ_France_ret_1d__minus__BDX_Becton_Dickinson_ret_1d", "MS_MorganStanley_ret_5d", "VIX_Price_zscore_60d__div__VRP", "MSTR_Bitcoin3_ret_5d", "EOG_EOGResources_ret_5d", "VIX_Price_zscore_60d__div__spx_vol_5d", "VIX_Price_zscore_60d__div__vix_vol_of_vol_5d", "ITT_ITTInc_ret_5d"], "is_new": false}, {"model_id": "v2_h1_CALM_RandomForest_N12", "algo": "RandomForest", "regime": "CALM", "horizon": 1, "n_features": 12, "F1_dir": 0.5458, "F1_UP_FORT": 0.1892, "F1_DOWN_FORT": 0.2885, "train_start": "2001-08-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VIX_Price_zscore_60d__ret5x__LLY_ret_1d", "hmm_p_stress__minus__spx_vol_5d", "XOM_ret_1d", "EWQ_France_ret_1d__minus__BDX_Becton_Dickinson_ret_1d", "MS_MorganStanley_ret_5d", "VIX_Price_zscore_60d__div__VRP", "MSTR_Bitcoin3_ret_5d", "EOG_EOGResources_ret_5d", "VIX_Price_zscore_60d__div__spx_vol_5d", "VIX_Price_zscore_60d__div__vix_vol_of_vol_5d", "ITT_ITTInc_ret_5d", "NVDA_vol_20d"], "is_new": false}, {"model_id": "v2_h1_CALM_RandomForest_N13", "algo": "RandomForest", "regime": "CALM", "horizon": 1, "n_features": 13, "F1_dir": 0.5737, "F1_UP_FORT": 0.1449, "F1_DOWN_FORT": 0.3186, "train_start": "2001-08-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VIX_Price_zscore_60d__ret5x__LLY_ret_1d", "hmm_p_stress__minus__spx_vol_5d", "XOM_ret_1d", "EWQ_France_ret_1d__minus__BDX_Becton_Dickinson_ret_1d", "MS_MorganStanley_ret_5d", "VIX_Price_zscore_60d__div__VRP", "MSTR_Bitcoin3_ret_5d", "EOG_EOGResources_ret_5d", "VIX_Price_zscore_60d__div__spx_vol_5d", "VIX_Price_zscore_60d__div__vix_vol_of_vol_5d", "ITT_ITTInc_ret_5d", "NVDA_vol_20d", "hmm_p_stress__zrel__spx_vol_5d"], "is_new": false}, {"model_id": "v2_h1_CALM_RandomForest_N14", "algo": "RandomForest", "regime": "CALM", "horizon": 1, "n_features": 14, "F1_dir": 0.5234, "F1_UP_FORT": 0.1918, "F1_DOWN_FORT": 0.3214, "train_start": "2001-08-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VIX_Price_zscore_60d__ret5x__LLY_ret_1d", "hmm_p_stress__minus__spx_vol_5d", "XOM_ret_1d", "EWQ_France_ret_1d__minus__BDX_Becton_Dickinson_ret_1d", "MS_MorganStanley_ret_5d", "VIX_Price_zscore_60d__div__VRP", "MSTR_Bitcoin3_ret_5d", "EOG_EOGResources_ret_5d", "VIX_Price_zscore_60d__div__spx_vol_5d", "VIX_Price_zscore_60d__div__vix_vol_of_vol_5d", "ITT_ITTInc_ret_5d", "NVDA_vol_20d", "hmm_p_stress__zrel__spx_vol_5d", "MS_MorganStanley_zscore_60d"], "is_new": false}, {"model_id": "v2_h1_CALM_RandomForest_N15", "algo": "RandomForest", "regime": "CALM", "horizon": 1, "n_features": 15, "F1_dir": 0.5501, "F1_UP_FORT": 0.2, "F1_DOWN_FORT": 0.3333, "train_start": "2001-08-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VIX_Price_zscore_60d__ret5x__LLY_ret_1d", "hmm_p_stress__minus__spx_vol_5d", "XOM_ret_1d", "EWQ_France_ret_1d__minus__BDX_Becton_Dickinson_ret_1d", "MS_MorganStanley_ret_5d", "VIX_Price_zscore_60d__div__VRP", "MSTR_Bitcoin3_ret_5d", "EOG_EOGResources_ret_5d", "VIX_Price_zscore_60d__div__spx_vol_5d", "VIX_Price_zscore_60d__div__vix_vol_of_vol_5d", "ITT_ITTInc_ret_5d", "NVDA_vol_20d", "hmm_p_stress__zrel__spx_vol_5d", "MS_MorganStanley_zscore_60d", "STLFSI4_ret_1d__ret5x__VRP"], "is_new": false}, {"model_id": "v2_h1_CALM_RandomForest_N16", "algo": "RandomForest", "regime": "CALM", "horizon": 1, "n_features": 16, "F1_dir": 0.5553, "F1_UP_FORT": 0.2133, "F1_DOWN_FORT": 0.3455, "train_start": "2001-08-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VIX_Price_zscore_60d__ret5x__LLY_ret_1d", "hmm_p_stress__minus__spx_vol_5d", "XOM_ret_1d", "EWQ_France_ret_1d__minus__BDX_Becton_Dickinson_ret_1d", "MS_MorganStanley_ret_5d", "VIX_Price_zscore_60d__div__VRP", "MSTR_Bitcoin3_ret_5d", "EOG_EOGResources_ret_5d", "VIX_Price_zscore_60d__div__spx_vol_5d", "VIX_Price_zscore_60d__div__vix_vol_of_vol_5d", "ITT_ITTInc_ret_5d", "NVDA_vol_20d", "hmm_p_stress__zrel__spx_vol_5d", "MS_MorganStanley_zscore_60d", "STLFSI4_ret_1d__ret5x__VRP", "AMGN_Amgen_ret_1d"], "is_new": false}, {"model_id": "v2_h1_CALM_RandomForest_N17", "algo": "RandomForest", "regime": "CALM", "horizon": 1, "n_features": 17, "F1_dir": 0.5941, "F1_UP_FORT": 0.1176, "F1_DOWN_FORT": 0.3333, "train_start": "2001-08-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VIX_Price_zscore_60d__ret5x__LLY_ret_1d", "hmm_p_stress__minus__spx_vol_5d", "XOM_ret_1d", "EWQ_France_ret_1d__minus__BDX_Becton_Dickinson_ret_1d", "MS_MorganStanley_ret_5d", "VIX_Price_zscore_60d__div__VRP", "MSTR_Bitcoin3_ret_5d", "EOG_EOGResources_ret_5d", "VIX_Price_zscore_60d__div__spx_vol_5d", "VIX_Price_zscore_60d__div__vix_vol_of_vol_5d", "ITT_ITTInc_ret_5d", "NVDA_vol_20d", "hmm_p_stress__zrel__spx_vol_5d", "MS_MorganStanley_zscore_60d", "STLFSI4_ret_1d__ret5x__VRP", "AMGN_Amgen_ret_1d", "NFCI_ret_5d"], "is_new": false}, {"model_id": "v2_h1_CALM_RandomForest_N18", "algo": "RandomForest", "regime": "CALM", "horizon": 1, "n_features": 18, "F1_dir": 0.5473, "F1_UP_FORT": 0.1918, "F1_DOWN_FORT": 0.3178, "train_start": "2001-08-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VIX_Price_zscore_60d__ret5x__LLY_ret_1d", "hmm_p_stress__minus__spx_vol_5d", "XOM_ret_1d", "EWQ_France_ret_1d__minus__BDX_Becton_Dickinson_ret_1d", "MS_MorganStanley_ret_5d", "VIX_Price_zscore_60d__div__VRP", "MSTR_Bitcoin3_ret_5d", "EOG_EOGResources_ret_5d", "VIX_Price_zscore_60d__div__spx_vol_5d", "VIX_Price_zscore_60d__div__vix_vol_of_vol_5d", "ITT_ITTInc_ret_5d", "NVDA_vol_20d", "hmm_p_stress__zrel__spx_vol_5d", "MS_MorganStanley_zscore_60d", "STLFSI4_ret_1d__ret5x__VRP", "AMGN_Amgen_ret_1d", "NFCI_ret_5d", "STLFSI4_ret_1d__minus__spx_vol_5d"], "is_new": false}, {"model_id": "v2_h1_CALM_RandomForest_N19", "algo": "RandomForest", "regime": "CALM", "horizon": 1, "n_features": 19, "F1_dir": 0.5623, "F1_UP_FORT": 0.1739, "F1_DOWN_FORT": 0.3186, "train_start": "2001-08-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VIX_Price_zscore_60d__ret5x__LLY_ret_1d", "hmm_p_stress__minus__spx_vol_5d", "XOM_ret_1d", "EWQ_France_ret_1d__minus__BDX_Becton_Dickinson_ret_1d", "MS_MorganStanley_ret_5d", "VIX_Price_zscore_60d__div__VRP", "MSTR_Bitcoin3_ret_5d", "EOG_EOGResources_ret_5d", "VIX_Price_zscore_60d__div__spx_vol_5d", "VIX_Price_zscore_60d__div__vix_vol_of_vol_5d", "ITT_ITTInc_ret_5d", "NVDA_vol_20d", "hmm_p_stress__zrel__spx_vol_5d", "MS_MorganStanley_zscore_60d", "STLFSI4_ret_1d__ret5x__VRP", "AMGN_Amgen_ret_1d", "NFCI_ret_5d", "STLFSI4_ret_1d__minus__spx_vol_5d", "STLFSI4_ret_1d__macross__BDX_Becton_Dickinson_ret_1d"], "is_new": false}, {"model_id": "v2_h1_CALM_LogisticRegression_N5", "algo": "LogisticRegression", "regime": "CALM", "horizon": 1, "n_features": 5, "F1_dir": 0.5446, "F1_UP_FORT": 0.1892, "F1_DOWN_FORT": 0.3301, "train_start": "2001-08-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VIX_Price_zscore_60d__ret5x__LLY_ret_1d", "hmm_p_stress__minus__spx_vol_5d", "XOM_ret_1d", "EWQ_France_ret_1d__minus__BDX_Becton_Dickinson_ret_1d", "MS_MorganStanley_ret_5d"], "is_new": false}, {"model_id": "v2_h1_CALM_LogisticRegression_N6", "algo": "LogisticRegression", "regime": "CALM", "horizon": 1, "n_features": 6, "F1_dir": 0.5697, "F1_UP_FORT": 0.1795, "F1_DOWN_FORT": 0.3469, "train_start": "2001-08-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VIX_Price_zscore_60d__ret5x__LLY_ret_1d", "hmm_p_stress__minus__spx_vol_5d", "XOM_ret_1d", "EWQ_France_ret_1d__minus__BDX_Becton_Dickinson_ret_1d", "MS_MorganStanley_ret_5d", "VIX_Price_zscore_60d__div__VRP"], "is_new": false}, {"model_id": "v2_h1_CALM_LogisticRegression_N7", "algo": "LogisticRegression", "regime": "CALM", "horizon": 1, "n_features": 7, "F1_dir": 0.5441, "F1_UP_FORT": 0.25, "F1_DOWN_FORT": 0.3409, "train_start": "2001-08-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VIX_Price_zscore_60d__ret5x__LLY_ret_1d", "hmm_p_stress__minus__spx_vol_5d", "XOM_ret_1d", "EWQ_France_ret_1d__minus__BDX_Becton_Dickinson_ret_1d", "MS_MorganStanley_ret_5d", "VIX_Price_zscore_60d__div__VRP", "MSTR_Bitcoin3_ret_5d"], "is_new": false}, {"model_id": "v2_h1_CALM_LogisticRegression_N8", "algo": "LogisticRegression", "regime": "CALM", "horizon": 1, "n_features": 8, "F1_dir": 0.5344, "F1_UP_FORT": 0.2245, "F1_DOWN_FORT": 0.3218, "train_start": "2001-08-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VIX_Price_zscore_60d__ret5x__LLY_ret_1d", "hmm_p_stress__minus__spx_vol_5d", "XOM_ret_1d", "EWQ_France_ret_1d__minus__BDX_Becton_Dickinson_ret_1d", "MS_MorganStanley_ret_5d", "VIX_Price_zscore_60d__div__VRP", "MSTR_Bitcoin3_ret_5d", "EOG_EOGResources_ret_5d"], "is_new": false}, {"model_id": "v2_h1_CALM_LogisticRegression_N9", "algo": "LogisticRegression", "regime": "CALM", "horizon": 1, "n_features": 9, "F1_dir": 0.5331, "F1_UP_FORT": 0.2308, "F1_DOWN_FORT": 0.3146, "train_start": "2001-08-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VIX_Price_zscore_60d__ret5x__LLY_ret_1d", "hmm_p_stress__minus__spx_vol_5d", "XOM_ret_1d", "EWQ_France_ret_1d__minus__BDX_Becton_Dickinson_ret_1d", "MS_MorganStanley_ret_5d", "VIX_Price_zscore_60d__div__VRP", "MSTR_Bitcoin3_ret_5d", "EOG_EOGResources_ret_5d", "VIX_Price_zscore_60d__div__spx_vol_5d"], "is_new": false}, {"model_id": "v2_h1_CALM_LogisticRegression_N10", "algo": "LogisticRegression", "regime": "CALM", "horizon": 1, "n_features": 10, "F1_dir": 0.542, "F1_UP_FORT": 0.2353, "F1_DOWN_FORT": 0.2989, "train_start": "2001-08-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VIX_Price_zscore_60d__ret5x__LLY_ret_1d", "hmm_p_stress__minus__spx_vol_5d", "XOM_ret_1d", "EWQ_France_ret_1d__minus__BDX_Becton_Dickinson_ret_1d", "MS_MorganStanley_ret_5d", "VIX_Price_zscore_60d__div__VRP", "MSTR_Bitcoin3_ret_5d", "EOG_EOGResources_ret_5d", "VIX_Price_zscore_60d__div__spx_vol_5d", "VIX_Price_zscore_60d__div__vix_vol_of_vol_5d"], "is_new": false}, {"model_id": "v2_h1_CALM_LogisticRegression_N11", "algo": "LogisticRegression", "regime": "CALM", "horizon": 1, "n_features": 11, "F1_dir": 0.5543, "F1_UP_FORT": 0.28, "F1_DOWN_FORT": 0.2955, "train_start": "2001-08-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VIX_Price_zscore_60d__ret5x__LLY_ret_1d", "hmm_p_stress__minus__spx_vol_5d", "XOM_ret_1d", "EWQ_France_ret_1d__minus__BDX_Becton_Dickinson_ret_1d", "MS_MorganStanley_ret_5d", "VIX_Price_zscore_60d__div__VRP", "MSTR_Bitcoin3_ret_5d", "EOG_EOGResources_ret_5d", "VIX_Price_zscore_60d__div__spx_vol_5d", "VIX_Price_zscore_60d__div__vix_vol_of_vol_5d", "ITT_ITTInc_ret_5d"], "is_new": false}, {"model_id": "v2_h1_CALM_LogisticRegression_N12", "algo": "LogisticRegression", "regime": "CALM", "horizon": 1, "n_features": 12, "F1_dir": 0.5459, "F1_UP_FORT": 0.2727, "F1_DOWN_FORT": 0.3011, "train_start": "2001-08-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VIX_Price_zscore_60d__ret5x__LLY_ret_1d", "hmm_p_stress__minus__spx_vol_5d", "XOM_ret_1d", "EWQ_France_ret_1d__minus__BDX_Becton_Dickinson_ret_1d", "MS_MorganStanley_ret_5d", "VIX_Price_zscore_60d__div__VRP", "MSTR_Bitcoin3_ret_5d", "EOG_EOGResources_ret_5d", "VIX_Price_zscore_60d__div__spx_vol_5d", "VIX_Price_zscore_60d__div__vix_vol_of_vol_5d", "ITT_ITTInc_ret_5d", "NVDA_vol_20d"], "is_new": false}, {"model_id": "v2_h1_CALM_LogisticRegression_N13", "algo": "LogisticRegression", "regime": "CALM", "horizon": 1, "n_features": 13, "F1_dir": 0.5553, "F1_UP_FORT": 0.2921, "F1_DOWN_FORT": 0.3441, "train_start": "2001-08-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VIX_Price_zscore_60d__ret5x__LLY_ret_1d", "hmm_p_stress__minus__spx_vol_5d", "XOM_ret_1d", "EWQ_France_ret_1d__minus__BDX_Becton_Dickinson_ret_1d", "MS_MorganStanley_ret_5d", "VIX_Price_zscore_60d__div__VRP", "MSTR_Bitcoin3_ret_5d", "EOG_EOGResources_ret_5d", "VIX_Price_zscore_60d__div__spx_vol_5d", "VIX_Price_zscore_60d__div__vix_vol_of_vol_5d", "ITT_ITTInc_ret_5d", "NVDA_vol_20d", "hmm_p_stress__zrel__spx_vol_5d"], "is_new": false}, {"model_id": "v2_h1_CALM_LogisticRegression_N14", "algo": "LogisticRegression", "regime": "CALM", "horizon": 1, "n_features": 14, "F1_dir": 0.5456, "F1_UP_FORT": 0.2889, "F1_DOWN_FORT": 0.3226, "train_start": "2001-08-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VIX_Price_zscore_60d__ret5x__LLY_ret_1d", "hmm_p_stress__minus__spx_vol_5d", "XOM_ret_1d", "EWQ_France_ret_1d__minus__BDX_Becton_Dickinson_ret_1d", "MS_MorganStanley_ret_5d", "VIX_Price_zscore_60d__div__VRP", "MSTR_Bitcoin3_ret_5d", "EOG_EOGResources_ret_5d", "VIX_Price_zscore_60d__div__spx_vol_5d", "VIX_Price_zscore_60d__div__vix_vol_of_vol_5d", "ITT_ITTInc_ret_5d", "NVDA_vol_20d", "hmm_p_stress__zrel__spx_vol_5d", "MS_MorganStanley_zscore_60d"], "is_new": false}, {"model_id": "v2_h1_CALM_LogisticRegression_N15", "algo": "LogisticRegression", "regime": "CALM", "horizon": 1, "n_features": 15, "F1_dir": 0.5553, "F1_UP_FORT": 0.2889, "F1_DOWN_FORT": 0.3191, "train_start": "2001-08-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VIX_Price_zscore_60d__ret5x__LLY_ret_1d", "hmm_p_stress__minus__spx_vol_5d", "XOM_ret_1d", "EWQ_France_ret_1d__minus__BDX_Becton_Dickinson_ret_1d", "MS_MorganStanley_ret_5d", "VIX_Price_zscore_60d__div__VRP", "MSTR_Bitcoin3_ret_5d", "EOG_EOGResources_ret_5d", "VIX_Price_zscore_60d__div__spx_vol_5d", "VIX_Price_zscore_60d__div__vix_vol_of_vol_5d", "ITT_ITTInc_ret_5d", "NVDA_vol_20d", "hmm_p_stress__zrel__spx_vol_5d", "MS_MorganStanley_zscore_60d", "STLFSI4_ret_1d__ret5x__VRP"], "is_new": false}, {"model_id": "v2_h1_CALM_LogisticRegression_N16", "algo": "LogisticRegression", "regime": "CALM", "horizon": 1, "n_features": 16, "F1_dir": 0.5507, "F1_UP_FORT": 0.25, "F1_DOWN_FORT": 0.3061, "train_start": "2001-08-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VIX_Price_zscore_60d__ret5x__LLY_ret_1d", "hmm_p_stress__minus__spx_vol_5d", "XOM_ret_1d", "EWQ_France_ret_1d__minus__BDX_Becton_Dickinson_ret_1d", "MS_MorganStanley_ret_5d", "VIX_Price_zscore_60d__div__VRP", "MSTR_Bitcoin3_ret_5d", "EOG_EOGResources_ret_5d", "VIX_Price_zscore_60d__div__spx_vol_5d", "VIX_Price_zscore_60d__div__vix_vol_of_vol_5d", "ITT_ITTInc_ret_5d", "NVDA_vol_20d", "hmm_p_stress__zrel__spx_vol_5d", "MS_MorganStanley_zscore_60d", "STLFSI4_ret_1d__ret5x__VRP", "AMGN_Amgen_ret_1d"], "is_new": false}, {"model_id": "v2_h1_CALM_LogisticRegression_N17", "algo": "LogisticRegression", "regime": "CALM", "horizon": 1, "n_features": 17, "F1_dir": 0.5383, "F1_UP_FORT": 0.225, "F1_DOWN_FORT": 0.3301, "train_start": "2001-08-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VIX_Price_zscore_60d__ret5x__LLY_ret_1d", "hmm_p_stress__minus__spx_vol_5d", "XOM_ret_1d", "EWQ_France_ret_1d__minus__BDX_Becton_Dickinson_ret_1d", "MS_MorganStanley_ret_5d", "VIX_Price_zscore_60d__div__VRP", "MSTR_Bitcoin3_ret_5d", "EOG_EOGResources_ret_5d", "VIX_Price_zscore_60d__div__spx_vol_5d", "VIX_Price_zscore_60d__div__vix_vol_of_vol_5d", "ITT_ITTInc_ret_5d", "NVDA_vol_20d", "hmm_p_stress__zrel__spx_vol_5d", "MS_MorganStanley_zscore_60d", "STLFSI4_ret_1d__ret5x__VRP", "AMGN_Amgen_ret_1d", "NFCI_ret_5d"], "is_new": false}, {"model_id": "v2_h1_CALM_LogisticRegression_N18", "algo": "LogisticRegression", "regime": "CALM", "horizon": 1, "n_features": 18, "F1_dir": 0.5338, "F1_UP_FORT": 0.25, "F1_DOWN_FORT": 0.32, "train_start": "2001-08-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VIX_Price_zscore_60d__ret5x__LLY_ret_1d", "hmm_p_stress__minus__spx_vol_5d", "XOM_ret_1d", "EWQ_France_ret_1d__minus__BDX_Becton_Dickinson_ret_1d", "MS_MorganStanley_ret_5d", "VIX_Price_zscore_60d__div__VRP", "MSTR_Bitcoin3_ret_5d", "EOG_EOGResources_ret_5d", "VIX_Price_zscore_60d__div__spx_vol_5d", "VIX_Price_zscore_60d__div__vix_vol_of_vol_5d", "ITT_ITTInc_ret_5d", "NVDA_vol_20d", "hmm_p_stress__zrel__spx_vol_5d", "MS_MorganStanley_zscore_60d", "STLFSI4_ret_1d__ret5x__VRP", "AMGN_Amgen_ret_1d", "NFCI_ret_5d", "STLFSI4_ret_1d__minus__spx_vol_5d"], "is_new": false}, {"model_id": "v2_h1_CALM_LogisticRegression_N19", "algo": "LogisticRegression", "regime": "CALM", "horizon": 1, "n_features": 19, "F1_dir": 0.5234, "F1_UP_FORT": 0.2025, "F1_DOWN_FORT": 0.3366, "train_start": "2001-08-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VIX_Price_zscore_60d__ret5x__LLY_ret_1d", "hmm_p_stress__minus__spx_vol_5d", "XOM_ret_1d", "EWQ_France_ret_1d__minus__BDX_Becton_Dickinson_ret_1d", "MS_MorganStanley_ret_5d", "VIX_Price_zscore_60d__div__VRP", "MSTR_Bitcoin3_ret_5d", "EOG_EOGResources_ret_5d", "VIX_Price_zscore_60d__div__spx_vol_5d", "VIX_Price_zscore_60d__div__vix_vol_of_vol_5d", "ITT_ITTInc_ret_5d", "NVDA_vol_20d", "hmm_p_stress__zrel__spx_vol_5d", "MS_MorganStanley_zscore_60d", "STLFSI4_ret_1d__ret5x__VRP", "AMGN_Amgen_ret_1d", "NFCI_ret_5d", "STLFSI4_ret_1d__minus__spx_vol_5d", "STLFSI4_ret_1d__macross__BDX_Becton_Dickinson_ret_1d"], "is_new": false}, {"model_id": "v2_h1_CALM_GradientBoosting_Optuna_N7", "algo": "GradientBoosting", "regime": "CALM", "horizon": 1, "n_features": 7, "F1_dir": 0.5987, "F1_UP_FORT": 0.1818, "F1_DOWN_FORT": 0.3711, "train_start": "2001-08-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VIX_Price_zscore_60d__ret5x__LLY_ret_1d", "hmm_p_stress__minus__spx_vol_5d", "XOM_ret_1d", "EWQ_France_ret_1d__minus__BDX_Becton_Dickinson_ret_1d", "MS_MorganStanley_ret_5d", "VIX_Price_zscore_60d__div__VRP", "MSTR_Bitcoin3_ret_5d"], "is_new": false}, {"model_id": "v2_h1_CALM_GradientBoosting_OptunaCal_N7", "algo": "GradientBoostingCal", "regime": "CALM", "horizon": 1, "n_features": 7, "F1_dir": 0.5942, "F1_UP_FORT": 0.1176, "F1_DOWN_FORT": 0.3958, "train_start": "2001-08-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VIX_Price_zscore_60d__ret5x__LLY_ret_1d", "hmm_p_stress__minus__spx_vol_5d", "XOM_ret_1d", "EWQ_France_ret_1d__minus__BDX_Becton_Dickinson_ret_1d", "MS_MorganStanley_ret_5d", "VIX_Price_zscore_60d__div__VRP", "MSTR_Bitcoin3_ret_5d"], "is_new": false}, {"model_id": "v2_h1_NORMAL_XGBoost_N5", "algo": "XGBoost", "regime": "NORMAL", "horizon": 1, "n_features": 5, "F1_dir": 0.5058, "F1_UP_FORT": 0.3051, "F1_DOWN_FORT": 0.3509, "train_start": "2000-11-02", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["heston_var_ev_h1__prod__heston_xi", "SBUX_vol_20d", "US5Y_Rate_ret_5d", "VIX_Price_zscore_60d__div__vix_vol_of_vol_10d", "VIX_Price_zscore_60d__minus__XLY_Disc_zscore_60d"], "is_new": false}, {"model_id": "v2_h1_NORMAL_XGBoost_N10", "algo": "XGBoost", "regime": "NORMAL", "horizon": 1, "n_features": 10, "F1_dir": 0.5026, "F1_UP_FORT": 0.2871, "F1_DOWN_FORT": 0.3261, "train_start": "2000-11-02", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["heston_var_ev_h1__prod__heston_xi", "SBUX_vol_20d", "US5Y_Rate_ret_5d", "VIX_Price_zscore_60d__div__vix_vol_of_vol_10d", "VIX_Price_zscore_60d__minus__XLY_Disc_zscore_60d", "EWY_Korea_ret_1d__prod__COF_CapitalOne_ret_5d", "XLY_Disc_zscore_60d__prod__BOVESPA_Brazil_ret_1d", "PLD_Prologis_ret_5d", "T_ret_1d__div__BOVESPA_Brazil_ret_1d", "BAC_ret_1d__prod__BOVESPA_Brazil_ret_1d"], "is_new": false}, {"model_id": "v2_h1_NORMAL_LightGBM_N5", "algo": "LightGBM", "regime": "NORMAL", "horizon": 1, "n_features": 5, "F1_dir": 0.5263, "F1_UP_FORT": 0.2971, "F1_DOWN_FORT": 0.3461, "train_start": "2000-11-02", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["heston_var_ev_h1__prod__heston_xi", "SBUX_vol_20d", "US5Y_Rate_ret_5d", "VIX_Price_zscore_60d__div__vix_vol_of_vol_10d", "VIX_Price_zscore_60d__minus__XLY_Disc_zscore_60d"], "is_new": false}, {"model_id": "v2_h1_NORMAL_LightGBM_N6", "algo": "LightGBM", "regime": "NORMAL", "horizon": 1, "n_features": 6, "F1_dir": 0.5268, "F1_UP_FORT": 0.2865, "F1_DOWN_FORT": 0.3491, "train_start": "2000-11-02", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["heston_var_ev_h1__prod__heston_xi", "SBUX_vol_20d", "US5Y_Rate_ret_5d", "VIX_Price_zscore_60d__div__vix_vol_of_vol_10d", "VIX_Price_zscore_60d__minus__XLY_Disc_zscore_60d", "EWY_Korea_ret_1d__prod__COF_CapitalOne_ret_5d"], "is_new": false}, {"model_id": "v2_h1_NORMAL_LightGBM_N7", "algo": "LightGBM", "regime": "NORMAL", "horizon": 1, "n_features": 7, "F1_dir": 0.5084, "F1_UP_FORT": 0.2993, "F1_DOWN_FORT": 0.2894, "train_start": "2000-11-02", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["heston_var_ev_h1__prod__heston_xi", "SBUX_vol_20d", "US5Y_Rate_ret_5d", "VIX_Price_zscore_60d__div__vix_vol_of_vol_10d", "VIX_Price_zscore_60d__minus__XLY_Disc_zscore_60d", "EWY_Korea_ret_1d__prod__COF_CapitalOne_ret_5d", "XLY_Disc_zscore_60d__prod__BOVESPA_Brazil_ret_1d"], "is_new": false}, {"model_id": "v2_h1_NORMAL_LightGBM_N10", "algo": "LightGBM", "regime": "NORMAL", "horizon": 1, "n_features": 10, "F1_dir": 0.504, "F1_UP_FORT": 0.2748, "F1_DOWN_FORT": 0.3294, "train_start": "2000-11-02", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["heston_var_ev_h1__prod__heston_xi", "SBUX_vol_20d", "US5Y_Rate_ret_5d", "VIX_Price_zscore_60d__div__vix_vol_of_vol_10d", "VIX_Price_zscore_60d__minus__XLY_Disc_zscore_60d", "EWY_Korea_ret_1d__prod__COF_CapitalOne_ret_5d", "XLY_Disc_zscore_60d__prod__BOVESPA_Brazil_ret_1d", "PLD_Prologis_ret_5d", "T_ret_1d__div__BOVESPA_Brazil_ret_1d", "BAC_ret_1d__prod__BOVESPA_Brazil_ret_1d"], "is_new": false}, {"model_id": "v2_h1_NORMAL_GradientBoosting_N5", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 1, "n_features": 5, "F1_dir": 0.5038, "F1_UP_FORT": 0.3007, "F1_DOWN_FORT": 0.3325, "train_start": "2000-11-02", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["heston_var_ev_h1__prod__heston_xi", "SBUX_vol_20d", "US5Y_Rate_ret_5d", "VIX_Price_zscore_60d__div__vix_vol_of_vol_10d", "VIX_Price_zscore_60d__minus__XLY_Disc_zscore_60d"], "is_new": false}, {"model_id": "v2_h1_NORMAL_GradientBoosting_N7", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 1, "n_features": 7, "F1_dir": 0.5094, "F1_UP_FORT": 0.2693, "F1_DOWN_FORT": 0.3367, "train_start": "2000-11-02", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["heston_var_ev_h1__prod__heston_xi", "SBUX_vol_20d", "US5Y_Rate_ret_5d", "VIX_Price_zscore_60d__div__vix_vol_of_vol_10d", "VIX_Price_zscore_60d__minus__XLY_Disc_zscore_60d", "EWY_Korea_ret_1d__prod__COF_CapitalOne_ret_5d", "XLY_Disc_zscore_60d__prod__BOVESPA_Brazil_ret_1d"], "is_new": false}, {"model_id": "v2_h1_NORMAL_GradientBoosting_N8", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 1, "n_features": 8, "F1_dir": 0.5226, "F1_UP_FORT": 0.2752, "F1_DOWN_FORT": 0.319, "train_start": "2000-11-02", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["heston_var_ev_h1__prod__heston_xi", "SBUX_vol_20d", "US5Y_Rate_ret_5d", "VIX_Price_zscore_60d__div__vix_vol_of_vol_10d", "VIX_Price_zscore_60d__minus__XLY_Disc_zscore_60d", "EWY_Korea_ret_1d__prod__COF_CapitalOne_ret_5d", "XLY_Disc_zscore_60d__prod__BOVESPA_Brazil_ret_1d", "PLD_Prologis_ret_5d"], "is_new": false}, {"model_id": "v2_h1_NORMAL_GradientBoosting_N10", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 1, "n_features": 10, "F1_dir": 0.504, "F1_UP_FORT": 0.2637, "F1_DOWN_FORT": 0.335, "train_start": "2000-11-02", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["heston_var_ev_h1__prod__heston_xi", "SBUX_vol_20d", "US5Y_Rate_ret_5d", "VIX_Price_zscore_60d__div__vix_vol_of_vol_10d", "VIX_Price_zscore_60d__minus__XLY_Disc_zscore_60d", "EWY_Korea_ret_1d__prod__COF_CapitalOne_ret_5d", "XLY_Disc_zscore_60d__prod__BOVESPA_Brazil_ret_1d", "PLD_Prologis_ret_5d", "T_ret_1d__div__BOVESPA_Brazil_ret_1d", "BAC_ret_1d__prod__BOVESPA_Brazil_ret_1d"], "is_new": false}, {"model_id": "v2_h1_NORMAL_RandomForest_N5", "algo": "RandomForest", "regime": "NORMAL", "horizon": 1, "n_features": 5, "F1_dir": 0.5395, "F1_UP_FORT": 0.3167, "F1_DOWN_FORT": 0.4103, "train_start": "2000-11-02", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["heston_var_ev_h1__prod__heston_xi", "SBUX_vol_20d", "US5Y_Rate_ret_5d", "VIX_Price_zscore_60d__div__vix_vol_of_vol_10d", "VIX_Price_zscore_60d__minus__XLY_Disc_zscore_60d"], "is_new": false}, {"model_id": "v2_h1_NORMAL_RandomForest_N6", "algo": "RandomForest", "regime": "NORMAL", "horizon": 1, "n_features": 6, "F1_dir": 0.5249, "F1_UP_FORT": 0.25, "F1_DOWN_FORT": 0.3883, "train_start": "2000-11-02", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["heston_var_ev_h1__prod__heston_xi", "SBUX_vol_20d", "US5Y_Rate_ret_5d", "VIX_Price_zscore_60d__div__vix_vol_of_vol_10d", "VIX_Price_zscore_60d__minus__XLY_Disc_zscore_60d", "EWY_Korea_ret_1d__prod__COF_CapitalOne_ret_5d"], "is_new": false}, {"model_id": "v2_h1_NORMAL_RandomForest_N7", "algo": "RandomForest", "regime": "NORMAL", "horizon": 1, "n_features": 7, "F1_dir": 0.5126, "F1_UP_FORT": 0.2663, "F1_DOWN_FORT": 0.3983, "train_start": "2000-11-02", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["heston_var_ev_h1__prod__heston_xi", "SBUX_vol_20d", "US5Y_Rate_ret_5d", "VIX_Price_zscore_60d__div__vix_vol_of_vol_10d", "VIX_Price_zscore_60d__minus__XLY_Disc_zscore_60d", "EWY_Korea_ret_1d__prod__COF_CapitalOne_ret_5d", "XLY_Disc_zscore_60d__prod__BOVESPA_Brazil_ret_1d"], "is_new": false}, {"model_id": "v2_h1_NORMAL_RandomForest_N8", "algo": "RandomForest", "regime": "NORMAL", "horizon": 1, "n_features": 8, "F1_dir": 0.5287, "F1_UP_FORT": 0.3021, "F1_DOWN_FORT": 0.3864, "train_start": "2000-11-02", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["heston_var_ev_h1__prod__heston_xi", "SBUX_vol_20d", "US5Y_Rate_ret_5d", "VIX_Price_zscore_60d__div__vix_vol_of_vol_10d", "VIX_Price_zscore_60d__minus__XLY_Disc_zscore_60d", "EWY_Korea_ret_1d__prod__COF_CapitalOne_ret_5d", "XLY_Disc_zscore_60d__prod__BOVESPA_Brazil_ret_1d", "PLD_Prologis_ret_5d"], "is_new": false}, {"model_id": "v2_h1_NORMAL_RandomForest_N10", "algo": "RandomForest", "regime": "NORMAL", "horizon": 1, "n_features": 10, "F1_dir": 0.5168, "F1_UP_FORT": 0.3089, "F1_DOWN_FORT": 0.3967, "train_start": "2000-11-02", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["heston_var_ev_h1__prod__heston_xi", "SBUX_vol_20d", "US5Y_Rate_ret_5d", "VIX_Price_zscore_60d__div__vix_vol_of_vol_10d", "VIX_Price_zscore_60d__minus__XLY_Disc_zscore_60d", "EWY_Korea_ret_1d__prod__COF_CapitalOne_ret_5d", "XLY_Disc_zscore_60d__prod__BOVESPA_Brazil_ret_1d", "PLD_Prologis_ret_5d", "T_ret_1d__div__BOVESPA_Brazil_ret_1d", "BAC_ret_1d__prod__BOVESPA_Brazil_ret_1d"], "is_new": false}, {"model_id": "v2_h1_NORMAL_LogisticRegression_N5", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 1, "n_features": 5, "F1_dir": 0.5265, "F1_UP_FORT": 0.267, "F1_DOWN_FORT": 0.396, "train_start": "2000-11-02", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["heston_var_ev_h1__prod__heston_xi", "SBUX_vol_20d", "US5Y_Rate_ret_5d", "VIX_Price_zscore_60d__div__vix_vol_of_vol_10d", "VIX_Price_zscore_60d__minus__XLY_Disc_zscore_60d"], "is_new": false}, {"model_id": "v2_h1_NORMAL_LogisticRegression_N6", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 1, "n_features": 6, "F1_dir": 0.5179, "F1_UP_FORT": 0.2822, "F1_DOWN_FORT": 0.3939, "train_start": "2000-11-02", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["heston_var_ev_h1__prod__heston_xi", "SBUX_vol_20d", "US5Y_Rate_ret_5d", "VIX_Price_zscore_60d__div__vix_vol_of_vol_10d", "VIX_Price_zscore_60d__minus__XLY_Disc_zscore_60d", "EWY_Korea_ret_1d__prod__COF_CapitalOne_ret_5d"], "is_new": false}, {"model_id": "v2_h1_NORMAL_LogisticRegression_N7", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 1, "n_features": 7, "F1_dir": 0.5135, "F1_UP_FORT": 0.2689, "F1_DOWN_FORT": 0.3773, "train_start": "2000-11-02", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["heston_var_ev_h1__prod__heston_xi", "SBUX_vol_20d", "US5Y_Rate_ret_5d", "VIX_Price_zscore_60d__div__vix_vol_of_vol_10d", "VIX_Price_zscore_60d__minus__XLY_Disc_zscore_60d", "EWY_Korea_ret_1d__prod__COF_CapitalOne_ret_5d", "XLY_Disc_zscore_60d__prod__BOVESPA_Brazil_ret_1d"], "is_new": false}, {"model_id": "v2_h1_NORMAL_LogisticRegression_N8", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 1, "n_features": 8, "F1_dir": 0.5007, "F1_UP_FORT": 0.2644, "F1_DOWN_FORT": 0.3618, "train_start": "2000-11-02", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["heston_var_ev_h1__prod__heston_xi", "SBUX_vol_20d", "US5Y_Rate_ret_5d", "VIX_Price_zscore_60d__div__vix_vol_of_vol_10d", "VIX_Price_zscore_60d__minus__XLY_Disc_zscore_60d", "EWY_Korea_ret_1d__prod__COF_CapitalOne_ret_5d", "XLY_Disc_zscore_60d__prod__BOVESPA_Brazil_ret_1d", "PLD_Prologis_ret_5d"], "is_new": false}, {"model_id": "v2_h1_NORMAL_LogisticRegression_N9", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 1, "n_features": 9, "F1_dir": 0.5036, "F1_UP_FORT": 0.2542, "F1_DOWN_FORT": 0.3575, "train_start": "2000-11-02", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["heston_var_ev_h1__prod__heston_xi", "SBUX_vol_20d", "US5Y_Rate_ret_5d", "VIX_Price_zscore_60d__div__vix_vol_of_vol_10d", "VIX_Price_zscore_60d__minus__XLY_Disc_zscore_60d", "EWY_Korea_ret_1d__prod__COF_CapitalOne_ret_5d", "XLY_Disc_zscore_60d__prod__BOVESPA_Brazil_ret_1d", "PLD_Prologis_ret_5d", "T_ret_1d__div__BOVESPA_Brazil_ret_1d"], "is_new": false}, {"model_id": "v2_h1_NORMAL_LogisticRegression_N10", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 1, "n_features": 10, "F1_dir": 0.5035, "F1_UP_FORT": 0.2682, "F1_DOWN_FORT": 0.3526, "train_start": "2000-11-02", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["heston_var_ev_h1__prod__heston_xi", "SBUX_vol_20d", "US5Y_Rate_ret_5d", "VIX_Price_zscore_60d__div__vix_vol_of_vol_10d", "VIX_Price_zscore_60d__minus__XLY_Disc_zscore_60d", "EWY_Korea_ret_1d__prod__COF_CapitalOne_ret_5d", "XLY_Disc_zscore_60d__prod__BOVESPA_Brazil_ret_1d", "PLD_Prologis_ret_5d", "T_ret_1d__div__BOVESPA_Brazil_ret_1d", "BAC_ret_1d__prod__BOVESPA_Brazil_ret_1d"], "is_new": false}, {"model_id": "v2_h1_NORMAL_RandomForest_OptunaCal_N8", "algo": "RandomForestCal", "regime": "NORMAL", "horizon": 1, "n_features": 8, "F1_dir": 0.5034, "F1_UP_FORT": 0.2582, "F1_DOWN_FORT": 0.2899, "train_start": "2000-11-02", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["heston_var_ev_h1__prod__heston_xi", "SBUX_vol_20d", "US5Y_Rate_ret_5d", "VIX_Price_zscore_60d__div__vix_vol_of_vol_10d", "VIX_Price_zscore_60d__minus__XLY_Disc_zscore_60d", "EWY_Korea_ret_1d__prod__COF_CapitalOne_ret_5d", "XLY_Disc_zscore_60d__prod__BOVESPA_Brazil_ret_1d", "PLD_Prologis_ret_5d"], "is_new": false}, {"model_id": "v2_h1_STRESS_XGBoost_N5", "algo": "XGBoost", "regime": "STRESS", "horizon": 1, "n_features": 5, "F1_dir": 0.5325, "F1_UP_FORT": 0.2182, "F1_DOWN_FORT": 0.4963, "train_start": "2001-02-06", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["heston_xi__prod__hmm_p_stress", "LOGI_Logitech_ret_1d__prod__CPB_CampbellSoup_ret_1d", "HangSeng_HK_vol_20d", "heston_var_ev_h7", "NFCI_ret_1d__macross__kalman_innovation"], "is_new": false}, {"model_id": "v2_h1_STRESS_XGBoost_N6", "algo": "XGBoost", "regime": "STRESS", "horizon": 1, "n_features": 6, "F1_dir": 0.5185, "F1_UP_FORT": 0.2078, "F1_DOWN_FORT": 0.4762, "train_start": "2001-02-06", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["heston_xi__prod__hmm_p_stress", "LOGI_Logitech_ret_1d__prod__CPB_CampbellSoup_ret_1d", "HangSeng_HK_vol_20d", "heston_var_ev_h7", "NFCI_ret_1d__macross__kalman_innovation", "XLY_Disc_ret_5d__prod__CPB_CampbellSoup_ret_1d"], "is_new": false}, {"model_id": "v2_h1_STRESS_XGBoost_N7", "algo": "XGBoost", "regime": "STRESS", "horizon": 1, "n_features": 7, "F1_dir": 0.5023, "F1_UP_FORT": 0.2105, "F1_DOWN_FORT": 0.4494, "train_start": "2001-02-06", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["heston_xi__prod__hmm_p_stress", "LOGI_Logitech_ret_1d__prod__CPB_CampbellSoup_ret_1d", "HangSeng_HK_vol_20d", "heston_var_ev_h7", "NFCI_ret_1d__macross__kalman_innovation", "XLY_Disc_ret_5d__prod__CPB_CampbellSoup_ret_1d", "US3Y_Rate_ret_5d"], "is_new": false}, {"model_id": "v2_h1_STRESS_XGBoost_N8", "algo": "XGBoost", "regime": "STRESS", "horizon": 1, "n_features": 8, "F1_dir": 0.5201, "F1_UP_FORT": 0.2222, "F1_DOWN_FORT": 0.4403, "train_start": "2001-02-06", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["heston_xi__prod__hmm_p_stress", "LOGI_Logitech_ret_1d__prod__CPB_CampbellSoup_ret_1d", "HangSeng_HK_vol_20d", "heston_var_ev_h7", "NFCI_ret_1d__macross__kalman_innovation", "XLY_Disc_ret_5d__prod__CPB_CampbellSoup_ret_1d", "US3Y_Rate_ret_5d", "BA_ret_1d"], "is_new": false}, {"model_id": "v2_h1_STRESS_XGBoost_N9", "algo": "XGBoost", "regime": "STRESS", "horizon": 1, "n_features": 9, "F1_dir": 0.5202, "F1_UP_FORT": 0.2208, "F1_DOWN_FORT": 0.4552, "train_start": "2001-02-06", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["heston_xi__prod__hmm_p_stress", "LOGI_Logitech_ret_1d__prod__CPB_CampbellSoup_ret_1d", "HangSeng_HK_vol_20d", "heston_var_ev_h7", "NFCI_ret_1d__macross__kalman_innovation", "XLY_Disc_ret_5d__prod__CPB_CampbellSoup_ret_1d", "US3Y_Rate_ret_5d", "BA_ret_1d", "BLK_BlackRock_zscore_60d"], "is_new": false}, {"model_id": "v2_h1_STRESS_XGBoost_N10", "algo": "XGBoost", "regime": "STRESS", "horizon": 1, "n_features": 10, "F1_dir": 0.5137, "F1_UP_FORT": 0.2297, "F1_DOWN_FORT": 0.4539, "train_start": "2001-02-06", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["heston_xi__prod__hmm_p_stress", "LOGI_Logitech_ret_1d__prod__CPB_CampbellSoup_ret_1d", "HangSeng_HK_vol_20d", "heston_var_ev_h7", "NFCI_ret_1d__macross__kalman_innovation", "XLY_Disc_ret_5d__prod__CPB_CampbellSoup_ret_1d", "US3Y_Rate_ret_5d", "BA_ret_1d", "BLK_BlackRock_zscore_60d", "PG_ret_1d__div__vix_vol_of_vol_5d"], "is_new": false}, {"model_id": "v2_h1_STRESS_XGBoost_N11", "algo": "XGBoost", "regime": "STRESS", "horizon": 1, "n_features": 11, "F1_dir": 0.5404, "F1_UP_FORT": 0.2803, "F1_DOWN_FORT": 0.493, "train_start": "2001-02-06", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["heston_xi__prod__hmm_p_stress", "LOGI_Logitech_ret_1d__prod__CPB_CampbellSoup_ret_1d", "HangSeng_HK_vol_20d", "heston_var_ev_h7", "NFCI_ret_1d__macross__kalman_innovation", "XLY_Disc_ret_5d__prod__CPB_CampbellSoup_ret_1d", "US3Y_Rate_ret_5d", "BA_ret_1d", "BLK_BlackRock_zscore_60d", "PG_ret_1d__div__vix_vol_of_vol_5d", "vix_vol_of_vol_5d__minus__XLY_Disc_ret_5d"], "is_new": false}, {"model_id": "v2_h1_STRESS_XGBoost_N12", "algo": "XGBoost", "regime": "STRESS", "horizon": 1, "n_features": 12, "F1_dir": 0.5299, "F1_UP_FORT": 0.1912, "F1_DOWN_FORT": 0.5185, "train_start": "2001-02-06", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["heston_xi__prod__hmm_p_stress", "LOGI_Logitech_ret_1d__prod__CPB_CampbellSoup_ret_1d", "HangSeng_HK_vol_20d", "heston_var_ev_h7", "NFCI_ret_1d__macross__kalman_innovation", "XLY_Disc_ret_5d__prod__CPB_CampbellSoup_ret_1d", "US3Y_Rate_ret_5d", "BA_ret_1d", "BLK_BlackRock_zscore_60d", "PG_ret_1d__div__vix_vol_of_vol_5d", "vix_vol_of_vol_5d__minus__XLY_Disc_ret_5d", "SCHW_Schwab_ret_1d__macross__XLY_Disc_ret_5d"], "is_new": false}, {"model_id": "v2_h1_STRESS_LightGBM_N5", "algo": "LightGBM", "regime": "STRESS", "horizon": 1, "n_features": 5, "F1_dir": 0.5107, "F1_UP_FORT": 0.253, "F1_DOWN_FORT": 0.4908, "train_start": "2001-02-06", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["heston_xi__prod__hmm_p_stress", "LOGI_Logitech_ret_1d__prod__CPB_CampbellSoup_ret_1d", "HangSeng_HK_vol_20d", "heston_var_ev_h7", "NFCI_ret_1d__macross__kalman_innovation"], "is_new": false}, {"model_id": "v2_h1_STRESS_LightGBM_N6", "algo": "LightGBM", "regime": "STRESS", "horizon": 1, "n_features": 6, "F1_dir": 0.5377, "F1_UP_FORT": 0.2609, "F1_DOWN_FORT": 0.4925, "train_start": "2001-02-06", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["heston_xi__prod__hmm_p_stress", "LOGI_Logitech_ret_1d__prod__CPB_CampbellSoup_ret_1d", "HangSeng_HK_vol_20d", "heston_var_ev_h7", "NFCI_ret_1d__macross__kalman_innovation", "XLY_Disc_ret_5d__prod__CPB_CampbellSoup_ret_1d"], "is_new": false}, {"model_id": "v2_h1_STRESS_LightGBM_N7", "algo": "LightGBM", "regime": "STRESS", "horizon": 1, "n_features": 7, "F1_dir": 0.5049, "F1_UP_FORT": 0.2609, "F1_DOWN_FORT": 0.4609, "train_start": "2001-02-06", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["heston_xi__prod__hmm_p_stress", "LOGI_Logitech_ret_1d__prod__CPB_CampbellSoup_ret_1d", "HangSeng_HK_vol_20d", "heston_var_ev_h7", "NFCI_ret_1d__macross__kalman_innovation", "XLY_Disc_ret_5d__prod__CPB_CampbellSoup_ret_1d", "US3Y_Rate_ret_5d"], "is_new": false}, {"model_id": "v2_h1_STRESS_LightGBM_N8", "algo": "LightGBM", "regime": "STRESS", "horizon": 1, "n_features": 8, "F1_dir": 0.5225, "F1_UP_FORT": 0.2532, "F1_DOWN_FORT": 0.4377, "train_start": "2001-02-06", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["heston_xi__prod__hmm_p_stress", "LOGI_Logitech_ret_1d__prod__CPB_CampbellSoup_ret_1d", "HangSeng_HK_vol_20d", "heston_var_ev_h7", "NFCI_ret_1d__macross__kalman_innovation", "XLY_Disc_ret_5d__prod__CPB_CampbellSoup_ret_1d", "US3Y_Rate_ret_5d", "BA_ret_1d"], "is_new": false}, {"model_id": "v2_h1_STRESS_LightGBM_N9", "algo": "LightGBM", "regime": "STRESS", "horizon": 1, "n_features": 9, "F1_dir": 0.5182, "F1_UP_FORT": 0.2771, "F1_DOWN_FORT": 0.4528, "train_start": "2001-02-06", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["heston_xi__prod__hmm_p_stress", "LOGI_Logitech_ret_1d__prod__CPB_CampbellSoup_ret_1d", "HangSeng_HK_vol_20d", "heston_var_ev_h7", "NFCI_ret_1d__macross__kalman_innovation", "XLY_Disc_ret_5d__prod__CPB_CampbellSoup_ret_1d", "US3Y_Rate_ret_5d", "BA_ret_1d", "BLK_BlackRock_zscore_60d"], "is_new": false}, {"model_id": "v2_h1_STRESS_LightGBM_N10", "algo": "LightGBM", "regime": "STRESS", "horizon": 1, "n_features": 10, "F1_dir": 0.5155, "F1_UP_FORT": 0.2683, "F1_DOWN_FORT": 0.4667, "train_start": "2001-02-06", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["heston_xi__prod__hmm_p_stress", "LOGI_Logitech_ret_1d__prod__CPB_CampbellSoup_ret_1d", "HangSeng_HK_vol_20d", "heston_var_ev_h7", "NFCI_ret_1d__macross__kalman_innovation", "XLY_Disc_ret_5d__prod__CPB_CampbellSoup_ret_1d", "US3Y_Rate_ret_5d", "BA_ret_1d", "BLK_BlackRock_zscore_60d", "PG_ret_1d__div__vix_vol_of_vol_5d"], "is_new": false}, {"model_id": "v2_h1_STRESS_LightGBM_N11", "algo": "LightGBM", "regime": "STRESS", "horizon": 1, "n_features": 11, "F1_dir": 0.5428, "F1_UP_FORT": 0.2517, "F1_DOWN_FORT": 0.4765, "train_start": "2001-02-06", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["heston_xi__prod__hmm_p_stress", "LOGI_Logitech_ret_1d__prod__CPB_CampbellSoup_ret_1d", "HangSeng_HK_vol_20d", "heston_var_ev_h7", "NFCI_ret_1d__macross__kalman_innovation", "XLY_Disc_ret_5d__prod__CPB_CampbellSoup_ret_1d", "US3Y_Rate_ret_5d", "BA_ret_1d", "BLK_BlackRock_zscore_60d", "PG_ret_1d__div__vix_vol_of_vol_5d", "vix_vol_of_vol_5d__minus__XLY_Disc_ret_5d"], "is_new": false}, {"model_id": "v2_h1_STRESS_LightGBM_N12", "algo": "LightGBM", "regime": "STRESS", "horizon": 1, "n_features": 12, "F1_dir": 0.5491, "F1_UP_FORT": 0.2781, "F1_DOWN_FORT": 0.5075, "train_start": "2001-02-06", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["heston_xi__prod__hmm_p_stress", "LOGI_Logitech_ret_1d__prod__CPB_CampbellSoup_ret_1d", "HangSeng_HK_vol_20d", "heston_var_ev_h7", "NFCI_ret_1d__macross__kalman_innovation", "XLY_Disc_ret_5d__prod__CPB_CampbellSoup_ret_1d", "US3Y_Rate_ret_5d", "BA_ret_1d", "BLK_BlackRock_zscore_60d", "PG_ret_1d__div__vix_vol_of_vol_5d", "vix_vol_of_vol_5d__minus__XLY_Disc_ret_5d", "SCHW_Schwab_ret_1d__macross__XLY_Disc_ret_5d"], "is_new": false}, {"model_id": "v2_h1_STRESS_GradientBoosting_N5", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 1, "n_features": 5, "F1_dir": 0.5076, "F1_UP_FORT": 0.242, "F1_DOWN_FORT": 0.4669, "train_start": "2001-02-06", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["heston_xi__prod__hmm_p_stress", "LOGI_Logitech_ret_1d__prod__CPB_CampbellSoup_ret_1d", "HangSeng_HK_vol_20d", "heston_var_ev_h7", "NFCI_ret_1d__macross__kalman_innovation"], "is_new": false}, {"model_id": "v2_h1_STRESS_GradientBoosting_N6", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 1, "n_features": 6, "F1_dir": 0.5304, "F1_UP_FORT": 0.239, "F1_DOWN_FORT": 0.4803, "train_start": "2001-02-06", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["heston_xi__prod__hmm_p_stress", "LOGI_Logitech_ret_1d__prod__CPB_CampbellSoup_ret_1d", "HangSeng_HK_vol_20d", "heston_var_ev_h7", "NFCI_ret_1d__macross__kalman_innovation", "XLY_Disc_ret_5d__prod__CPB_CampbellSoup_ret_1d"], "is_new": false}, {"model_id": "v2_h1_STRESS_GradientBoosting_N7", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 1, "n_features": 7, "F1_dir": 0.5116, "F1_UP_FORT": 0.2692, "F1_DOWN_FORT": 0.4436, "train_start": "2001-02-06", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["heston_xi__prod__hmm_p_stress", "LOGI_Logitech_ret_1d__prod__CPB_CampbellSoup_ret_1d", "HangSeng_HK_vol_20d", "heston_var_ev_h7", "NFCI_ret_1d__macross__kalman_innovation", "XLY_Disc_ret_5d__prod__CPB_CampbellSoup_ret_1d", "US3Y_Rate_ret_5d"], "is_new": false}, {"model_id": "v2_h1_STRESS_GradientBoosting_N8", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 1, "n_features": 8, "F1_dir": 0.5329, "F1_UP_FORT": 0.2945, "F1_DOWN_FORT": 0.4385, "train_start": "2001-02-06", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["heston_xi__prod__hmm_p_stress", "LOGI_Logitech_ret_1d__prod__CPB_CampbellSoup_ret_1d", "HangSeng_HK_vol_20d", "heston_var_ev_h7", "NFCI_ret_1d__macross__kalman_innovation", "XLY_Disc_ret_5d__prod__CPB_CampbellSoup_ret_1d", "US3Y_Rate_ret_5d", "BA_ret_1d"], "is_new": false}, {"model_id": "v2_h1_STRESS_GradientBoosting_N9", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 1, "n_features": 9, "F1_dir": 0.5521, "F1_UP_FORT": 0.2875, "F1_DOWN_FORT": 0.4758, "train_start": "2001-02-06", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["heston_xi__prod__hmm_p_stress", "LOGI_Logitech_ret_1d__prod__CPB_CampbellSoup_ret_1d", "HangSeng_HK_vol_20d", "heston_var_ev_h7", "NFCI_ret_1d__macross__kalman_innovation", "XLY_Disc_ret_5d__prod__CPB_CampbellSoup_ret_1d", "US3Y_Rate_ret_5d", "BA_ret_1d", "BLK_BlackRock_zscore_60d"], "is_new": false}, {"model_id": "v2_h1_STRESS_GradientBoosting_N10", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 1, "n_features": 10, "F1_dir": 0.5672, "F1_UP_FORT": 0.2667, "F1_DOWN_FORT": 0.4876, "train_start": "2001-02-06", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["heston_xi__prod__hmm_p_stress", "LOGI_Logitech_ret_1d__prod__CPB_CampbellSoup_ret_1d", "HangSeng_HK_vol_20d", "heston_var_ev_h7", "NFCI_ret_1d__macross__kalman_innovation", "XLY_Disc_ret_5d__prod__CPB_CampbellSoup_ret_1d", "US3Y_Rate_ret_5d", "BA_ret_1d", "BLK_BlackRock_zscore_60d", "PG_ret_1d__div__vix_vol_of_vol_5d"], "is_new": false}, {"model_id": "v2_h1_STRESS_GradientBoosting_N11", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 1, "n_features": 11, "F1_dir": 0.5241, "F1_UP_FORT": 0.2635, "F1_DOWN_FORT": 0.4765, "train_start": "2001-02-06", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["heston_xi__prod__hmm_p_stress", "LOGI_Logitech_ret_1d__prod__CPB_CampbellSoup_ret_1d", "HangSeng_HK_vol_20d", "heston_var_ev_h7", "NFCI_ret_1d__macross__kalman_innovation", "XLY_Disc_ret_5d__prod__CPB_CampbellSoup_ret_1d", "US3Y_Rate_ret_5d", "BA_ret_1d", "BLK_BlackRock_zscore_60d", "PG_ret_1d__div__vix_vol_of_vol_5d", "vix_vol_of_vol_5d__minus__XLY_Disc_ret_5d"], "is_new": false}, {"model_id": "v2_h1_STRESS_GradientBoosting_N12", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 1, "n_features": 12, "F1_dir": 0.5631, "F1_UP_FORT": 0.2302, "F1_DOWN_FORT": 0.4949, "train_start": "2001-02-06", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["heston_xi__prod__hmm_p_stress", "LOGI_Logitech_ret_1d__prod__CPB_CampbellSoup_ret_1d", "HangSeng_HK_vol_20d", "heston_var_ev_h7", "NFCI_ret_1d__macross__kalman_innovation", "XLY_Disc_ret_5d__prod__CPB_CampbellSoup_ret_1d", "US3Y_Rate_ret_5d", "BA_ret_1d", "BLK_BlackRock_zscore_60d", "PG_ret_1d__div__vix_vol_of_vol_5d", "vix_vol_of_vol_5d__minus__XLY_Disc_ret_5d", "SCHW_Schwab_ret_1d__macross__XLY_Disc_ret_5d"], "is_new": false}, {"model_id": "v2_h1_STRESS_RandomForest_N5", "algo": "RandomForest", "regime": "STRESS", "horizon": 1, "n_features": 5, "F1_dir": 0.5298, "F1_UP_FORT": 0.1527, "F1_DOWN_FORT": 0.5132, "train_start": "2001-02-06", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["heston_xi__prod__hmm_p_stress", "LOGI_Logitech_ret_1d__prod__CPB_CampbellSoup_ret_1d", "HangSeng_HK_vol_20d", "heston_var_ev_h7", "NFCI_ret_1d__macross__kalman_innovation"], "is_new": false}, {"model_id": "v2_h1_STRESS_RandomForest_N6", "algo": "RandomForest", "regime": "STRESS", "horizon": 1, "n_features": 6, "F1_dir": 0.5171, "F1_UP_FORT": 0.1, "F1_DOWN_FORT": 0.4984, "train_start": "2001-02-06", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["heston_xi__prod__hmm_p_stress", "LOGI_Logitech_ret_1d__prod__CPB_CampbellSoup_ret_1d", "HangSeng_HK_vol_20d", "heston_var_ev_h7", "NFCI_ret_1d__macross__kalman_innovation", "XLY_Disc_ret_5d__prod__CPB_CampbellSoup_ret_1d"], "is_new": false}, {"model_id": "v2_h1_STRESS_RandomForest_N7", "algo": "RandomForest", "regime": "STRESS", "horizon": 1, "n_features": 7, "F1_dir": 0.5098, "F1_UP_FORT": 0.1217, "F1_DOWN_FORT": 0.5115, "train_start": "2001-02-06", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["heston_xi__prod__hmm_p_stress", "LOGI_Logitech_ret_1d__prod__CPB_CampbellSoup_ret_1d", "HangSeng_HK_vol_20d", "heston_var_ev_h7", "NFCI_ret_1d__macross__kalman_innovation", "XLY_Disc_ret_5d__prod__CPB_CampbellSoup_ret_1d", "US3Y_Rate_ret_5d"], "is_new": false}, {"model_id": "v2_h1_STRESS_RandomForest_N8", "algo": "RandomForest", "regime": "STRESS", "horizon": 1, "n_features": 8, "F1_dir": 0.5256, "F1_UP_FORT": 0.1062, "F1_DOWN_FORT": 0.4967, "train_start": "2001-02-06", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["heston_xi__prod__hmm_p_stress", "LOGI_Logitech_ret_1d__prod__CPB_CampbellSoup_ret_1d", "HangSeng_HK_vol_20d", "heston_var_ev_h7", "NFCI_ret_1d__macross__kalman_innovation", "XLY_Disc_ret_5d__prod__CPB_CampbellSoup_ret_1d", "US3Y_Rate_ret_5d", "BA_ret_1d"], "is_new": false}, {"model_id": "v2_h1_STRESS_RandomForest_N9", "algo": "RandomForest", "regime": "STRESS", "horizon": 1, "n_features": 9, "F1_dir": 0.5271, "F1_UP_FORT": 0.1197, "F1_DOWN_FORT": 0.5096, "train_start": "2001-02-06", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["heston_xi__prod__hmm_p_stress", "LOGI_Logitech_ret_1d__prod__CPB_CampbellSoup_ret_1d", "HangSeng_HK_vol_20d", "heston_var_ev_h7", "NFCI_ret_1d__macross__kalman_innovation", "XLY_Disc_ret_5d__prod__CPB_CampbellSoup_ret_1d", "US3Y_Rate_ret_5d", "BA_ret_1d", "BLK_BlackRock_zscore_60d"], "is_new": false}, {"model_id": "v2_h1_STRESS_RandomForest_N10", "algo": "RandomForest", "regime": "STRESS", "horizon": 1, "n_features": 10, "F1_dir": 0.5189, "F1_UP_FORT": 0.1111, "F1_DOWN_FORT": 0.4935, "train_start": "2001-02-06", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["heston_xi__prod__hmm_p_stress", "LOGI_Logitech_ret_1d__prod__CPB_CampbellSoup_ret_1d", "HangSeng_HK_vol_20d", "heston_var_ev_h7", "NFCI_ret_1d__macross__kalman_innovation", "XLY_Disc_ret_5d__prod__CPB_CampbellSoup_ret_1d", "US3Y_Rate_ret_5d", "BA_ret_1d", "BLK_BlackRock_zscore_60d", "PG_ret_1d__div__vix_vol_of_vol_5d"], "is_new": false}, {"model_id": "v2_h1_STRESS_RandomForest_N11", "algo": "RandomForest", "regime": "STRESS", "horizon": 1, "n_features": 11, "F1_dir": 0.5171, "F1_UP_FORT": 0.087, "F1_DOWN_FORT": 0.5047, "train_start": "2001-02-06", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["heston_xi__prod__hmm_p_stress", "LOGI_Logitech_ret_1d__prod__CPB_CampbellSoup_ret_1d", "HangSeng_HK_vol_20d", "heston_var_ev_h7", "NFCI_ret_1d__macross__kalman_innovation", "XLY_Disc_ret_5d__prod__CPB_CampbellSoup_ret_1d", "US3Y_Rate_ret_5d", "BA_ret_1d", "BLK_BlackRock_zscore_60d", "PG_ret_1d__div__vix_vol_of_vol_5d", "vix_vol_of_vol_5d__minus__XLY_Disc_ret_5d"], "is_new": false}, {"model_id": "v2_h1_STRESS_RandomForest_N12", "algo": "RandomForest", "regime": "STRESS", "horizon": 1, "n_features": 12, "F1_dir": 0.5004, "F1_UP_FORT": 0.1681, "F1_DOWN_FORT": 0.5031, "train_start": "2001-02-06", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["heston_xi__prod__hmm_p_stress", "LOGI_Logitech_ret_1d__prod__CPB_CampbellSoup_ret_1d", "HangSeng_HK_vol_20d", "heston_var_ev_h7", "NFCI_ret_1d__macross__kalman_innovation", "XLY_Disc_ret_5d__prod__CPB_CampbellSoup_ret_1d", "US3Y_Rate_ret_5d", "BA_ret_1d", "BLK_BlackRock_zscore_60d", "PG_ret_1d__div__vix_vol_of_vol_5d", "vix_vol_of_vol_5d__minus__XLY_Disc_ret_5d", "SCHW_Schwab_ret_1d__macross__XLY_Disc_ret_5d"], "is_new": false}, {"model_id": "v2_h1_STRESS_LogisticRegression_N5", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 1, "n_features": 5, "F1_dir": 0.5275, "F1_UP_FORT": 0.1217, "F1_DOWN_FORT": 0.5356, "train_start": "2001-02-06", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["heston_xi__prod__hmm_p_stress", "LOGI_Logitech_ret_1d__prod__CPB_CampbellSoup_ret_1d", "HangSeng_HK_vol_20d", "heston_var_ev_h7", "NFCI_ret_1d__macross__kalman_innovation"], "is_new": false}, {"model_id": "v2_h1_STRESS_LogisticRegression_N6", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 1, "n_features": 6, "F1_dir": 0.519, "F1_UP_FORT": 0.1053, "F1_DOWN_FORT": 0.5267, "train_start": "2001-02-06", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["heston_xi__prod__hmm_p_stress", "LOGI_Logitech_ret_1d__prod__CPB_CampbellSoup_ret_1d", "HangSeng_HK_vol_20d", "heston_var_ev_h7", "NFCI_ret_1d__macross__kalman_innovation", "XLY_Disc_ret_5d__prod__CPB_CampbellSoup_ret_1d"], "is_new": false}, {"model_id": "v2_h1_STRESS_LogisticRegression_N7", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 1, "n_features": 7, "F1_dir": 0.532, "F1_UP_FORT": 0.0862, "F1_DOWN_FORT": 0.5205, "train_start": "2001-02-06", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["heston_xi__prod__hmm_p_stress", "LOGI_Logitech_ret_1d__prod__CPB_CampbellSoup_ret_1d", "HangSeng_HK_vol_20d", "heston_var_ev_h7", "NFCI_ret_1d__macross__kalman_innovation", "XLY_Disc_ret_5d__prod__CPB_CampbellSoup_ret_1d", "US3Y_Rate_ret_5d"], "is_new": false}, {"model_id": "v2_h1_STRESS_LogisticRegression_N8", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 1, "n_features": 8, "F1_dir": 0.5523, "F1_UP_FORT": 0.2014, "F1_DOWN_FORT": 0.5053, "train_start": "2001-02-06", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["heston_xi__prod__hmm_p_stress", "LOGI_Logitech_ret_1d__prod__CPB_CampbellSoup_ret_1d", "HangSeng_HK_vol_20d", "heston_var_ev_h7", "NFCI_ret_1d__macross__kalman_innovation", "XLY_Disc_ret_5d__prod__CPB_CampbellSoup_ret_1d", "US3Y_Rate_ret_5d", "BA_ret_1d"], "is_new": false}, {"model_id": "v2_h1_STRESS_LogisticRegression_N9", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 1, "n_features": 9, "F1_dir": 0.5405, "F1_UP_FORT": 0.2044, "F1_DOWN_FORT": 0.5084, "train_start": "2001-02-06", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["heston_xi__prod__hmm_p_stress", "LOGI_Logitech_ret_1d__prod__CPB_CampbellSoup_ret_1d", "HangSeng_HK_vol_20d", "heston_var_ev_h7", "NFCI_ret_1d__macross__kalman_innovation", "XLY_Disc_ret_5d__prod__CPB_CampbellSoup_ret_1d", "US3Y_Rate_ret_5d", "BA_ret_1d", "BLK_BlackRock_zscore_60d"], "is_new": false}, {"model_id": "v2_h1_STRESS_LogisticRegression_N10", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 1, "n_features": 10, "F1_dir": 0.5474, "F1_UP_FORT": 0.1884, "F1_DOWN_FORT": 0.5033, "train_start": "2001-02-06", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["heston_xi__prod__hmm_p_stress", "LOGI_Logitech_ret_1d__prod__CPB_CampbellSoup_ret_1d", "HangSeng_HK_vol_20d", "heston_var_ev_h7", "NFCI_ret_1d__macross__kalman_innovation", "XLY_Disc_ret_5d__prod__CPB_CampbellSoup_ret_1d", "US3Y_Rate_ret_5d", "BA_ret_1d", "BLK_BlackRock_zscore_60d", "PG_ret_1d__div__vix_vol_of_vol_5d"], "is_new": false}, {"model_id": "v2_h1_STRESS_LogisticRegression_N11", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 1, "n_features": 11, "F1_dir": 0.5608, "F1_UP_FORT": 0.1884, "F1_DOWN_FORT": 0.5137, "train_start": "2001-02-06", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["heston_xi__prod__hmm_p_stress", "LOGI_Logitech_ret_1d__prod__CPB_CampbellSoup_ret_1d", "HangSeng_HK_vol_20d", "heston_var_ev_h7", "NFCI_ret_1d__macross__kalman_innovation", "XLY_Disc_ret_5d__prod__CPB_CampbellSoup_ret_1d", "US3Y_Rate_ret_5d", "BA_ret_1d", "BLK_BlackRock_zscore_60d", "PG_ret_1d__div__vix_vol_of_vol_5d", "vix_vol_of_vol_5d__minus__XLY_Disc_ret_5d"], "is_new": false}, {"model_id": "v2_h1_STRESS_LogisticRegression_N12", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 1, "n_features": 12, "F1_dir": 0.5745, "F1_UP_FORT": 0.1972, "F1_DOWN_FORT": 0.526, "train_start": "2001-02-06", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["heston_xi__prod__hmm_p_stress", "LOGI_Logitech_ret_1d__prod__CPB_CampbellSoup_ret_1d", "HangSeng_HK_vol_20d", "heston_var_ev_h7", "NFCI_ret_1d__macross__kalman_innovation", "XLY_Disc_ret_5d__prod__CPB_CampbellSoup_ret_1d", "US3Y_Rate_ret_5d", "BA_ret_1d", "BLK_BlackRock_zscore_60d", "PG_ret_1d__div__vix_vol_of_vol_5d", "vix_vol_of_vol_5d__minus__XLY_Disc_ret_5d", "SCHW_Schwab_ret_1d__macross__XLY_Disc_ret_5d"], "is_new": false}, {"model_id": "v2_h1_STRESS_LightGBM_Optuna_N12", "algo": "LightGBM", "regime": "STRESS", "horizon": 1, "n_features": 12, "F1_dir": 0.5295, "F1_UP_FORT": 0.2553, "F1_DOWN_FORT": 0.4848, "train_start": "2001-02-06", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["heston_xi__prod__hmm_p_stress", "LOGI_Logitech_ret_1d__prod__CPB_CampbellSoup_ret_1d", "HangSeng_HK_vol_20d", "heston_var_ev_h7", "NFCI_ret_1d__macross__kalman_innovation", "XLY_Disc_ret_5d__prod__CPB_CampbellSoup_ret_1d", "US3Y_Rate_ret_5d", "BA_ret_1d", "BLK_BlackRock_zscore_60d", "PG_ret_1d__div__vix_vol_of_vol_5d", "vix_vol_of_vol_5d__minus__XLY_Disc_ret_5d", "SCHW_Schwab_ret_1d__macross__XLY_Disc_ret_5d"], "is_new": false}, {"model_id": "v2_h1_STRESS_LightGBM_OptunaCal_N12", "algo": "LightGBMCal", "regime": "STRESS", "horizon": 1, "n_features": 12, "F1_dir": 0.5375, "F1_UP_FORT": 0.2838, "F1_DOWN_FORT": 0.4842, "train_start": "2001-02-06", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["heston_xi__prod__hmm_p_stress", "LOGI_Logitech_ret_1d__prod__CPB_CampbellSoup_ret_1d", "HangSeng_HK_vol_20d", "heston_var_ev_h7", "NFCI_ret_1d__macross__kalman_innovation", "XLY_Disc_ret_5d__prod__CPB_CampbellSoup_ret_1d", "US3Y_Rate_ret_5d", "BA_ret_1d", "BLK_BlackRock_zscore_60d", "PG_ret_1d__div__vix_vol_of_vol_5d", "vix_vol_of_vol_5d__minus__XLY_Disc_ret_5d", "SCHW_Schwab_ret_1d__macross__XLY_Disc_ret_5d"], "is_new": false}, {"model_id": "v2_h1_GLOBAL_LogisticRegression_N5", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 1, "n_features": 5, "F1_dir": 0.5376, "F1_UP_FORT": 0.211, "F1_DOWN_FORT": 0.4198, "train_start": "2001-02-06", "sampler": "SMOTETomek", "best_params": "{}", "features": ["kalman_innovation__div__kalman_residual", "heston_xi__div__kalman_residual", "heston_xi__div__kalman_innovation", "kalman_innovation__zrel__kalman_residual", "kalman_innovation__prod__kalman_residual"], "is_new": false}, {"model_id": "v2_h1_GLOBAL_LogisticRegression_N6", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 1, "n_features": 6, "F1_dir": 0.536, "F1_UP_FORT": 0.1992, "F1_DOWN_FORT": 0.4223, "train_start": "2001-02-06", "sampler": "SMOTETomek", "best_params": "{}", "features": ["kalman_innovation__div__kalman_residual", "heston_xi__div__kalman_residual", "heston_xi__div__kalman_innovation", "kalman_innovation__zrel__kalman_residual", "kalman_innovation__prod__kalman_residual", "kalman_innovation__minus__kalman_residual"], "is_new": false}, {"model_id": "v2_h1_GLOBAL_LogisticRegression_N7", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 1, "n_features": 7, "F1_dir": 0.5428, "F1_UP_FORT": 0.2055, "F1_DOWN_FORT": 0.4152, "train_start": "2001-02-06", "sampler": "SMOTETomek", "best_params": "{}", "features": ["kalman_innovation__div__kalman_residual", "heston_xi__div__kalman_residual", "heston_xi__div__kalman_innovation", "kalman_innovation__zrel__kalman_residual", "kalman_innovation__prod__kalman_residual", "kalman_innovation__minus__kalman_residual", "kalman_innovation__macross__kalman_residual"], "is_new": false}, {"model_id": "v2_h1_GLOBAL_LogisticRegression_N8", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 1, "n_features": 8, "F1_dir": 0.5747, "F1_UP_FORT": 0.2218, "F1_DOWN_FORT": 0.3916, "train_start": "2001-02-06", "sampler": "SMOTETomek", "best_params": "{}", "features": ["kalman_innovation__div__kalman_residual", "heston_xi__div__kalman_residual", "heston_xi__div__kalman_innovation", "kalman_innovation__zrel__kalman_residual", "kalman_innovation__prod__kalman_residual", "kalman_innovation__minus__kalman_residual", "kalman_innovation__macross__kalman_residual", "DIS_vol_20d"], "is_new": false}, {"model_id": "v2_h3_CALM_XGBoost_N7", "algo": "XGBoost", "regime": "CALM", "horizon": 3, "n_features": 7, "F1_dir": 0.5151, "F1_UP_FORT": 0.1818, "F1_DOWN_FORT": 0.3415, "train_start": "2000-11-02", "sampler": "SMOTE", "best_params": "{}", "features": ["WTI_Oil_FRED_ret_5d__minus__vix_max_abs_ret_5d", "NFCI_ret_5d__prod__hmm_p_stress", "VIX_Price_zscore_60d__div__MSTR_Bitcoin3_ret_5d", "AORD_AUS_ret_5d__minus__spx_drawdown_252d", "EWQ_France_zscore_60d", "EWQ_France_ret_20d", "DHR_ret_1d"], "is_new": false}, {"model_id": "v2_h3_CALM_XGBoost_N8", "algo": "XGBoost", "regime": "CALM", "horizon": 3, "n_features": 8, "F1_dir": 0.5195, "F1_UP_FORT": 0.186, "F1_DOWN_FORT": 0.3065, "train_start": "2000-11-02", "sampler": "SMOTE", "best_params": "{}", "features": ["WTI_Oil_FRED_ret_5d__minus__vix_max_abs_ret_5d", "NFCI_ret_5d__prod__hmm_p_stress", "VIX_Price_zscore_60d__div__MSTR_Bitcoin3_ret_5d", "AORD_AUS_ret_5d__minus__spx_drawdown_252d", "EWQ_France_zscore_60d", "EWQ_France_ret_20d", "DHR_ret_1d", "MRK_Merck_zscore_60d"], "is_new": false}, {"model_id": "v2_h3_CALM_XGBoost_N9", "algo": "XGBoost", "regime": "CALM", "horizon": 3, "n_features": 9, "F1_dir": 0.5193, "F1_UP_FORT": 0.2353, "F1_DOWN_FORT": 0.3, "train_start": "2000-11-02", "sampler": "SMOTE", "best_params": "{}", "features": ["WTI_Oil_FRED_ret_5d__minus__vix_max_abs_ret_5d", "NFCI_ret_5d__prod__hmm_p_stress", "VIX_Price_zscore_60d__div__MSTR_Bitcoin3_ret_5d", "AORD_AUS_ret_5d__minus__spx_drawdown_252d", "EWQ_France_zscore_60d", "EWQ_France_ret_20d", "DHR_ret_1d", "MRK_Merck_zscore_60d", "LMT_LockheedMartin_vol_20d"], "is_new": false}, {"model_id": "v2_h3_CALM_XGBoost_N11", "algo": "XGBoost", "regime": "CALM", "horizon": 3, "n_features": 11, "F1_dir": 0.5325, "F1_UP_FORT": 0.26, "F1_DOWN_FORT": 0.3, "train_start": "2000-11-02", "sampler": "SMOTE", "best_params": "{}", "features": ["WTI_Oil_FRED_ret_5d__minus__vix_max_abs_ret_5d", "NFCI_ret_5d__prod__hmm_p_stress", "VIX_Price_zscore_60d__div__MSTR_Bitcoin3_ret_5d", "AORD_AUS_ret_5d__minus__spx_drawdown_252d", "EWQ_France_zscore_60d", "EWQ_France_ret_20d", "DHR_ret_1d", "MRK_Merck_zscore_60d", "LMT_LockheedMartin_vol_20d", "XLB_Materials_zscore_60d", "QQQ_vol_20d__prod__vix_max_abs_ret_5d"], "is_new": false}, {"model_id": "v2_h3_CALM_XGBoost_N12", "algo": "XGBoost", "regime": "CALM", "horizon": 3, "n_features": 12, "F1_dir": 0.5706, "F1_UP_FORT": 0.2828, "F1_DOWN_FORT": 0.3519, "train_start": "2000-11-02", "sampler": "SMOTE", "best_params": "{}", "features": ["WTI_Oil_FRED_ret_5d__minus__vix_max_abs_ret_5d", "NFCI_ret_5d__prod__hmm_p_stress", "VIX_Price_zscore_60d__div__MSTR_Bitcoin3_ret_5d", "AORD_AUS_ret_5d__minus__spx_drawdown_252d", "EWQ_France_zscore_60d", "EWQ_France_ret_20d", "DHR_ret_1d", "MRK_Merck_zscore_60d", "LMT_LockheedMartin_vol_20d", "XLB_Materials_zscore_60d", "QQQ_vol_20d__prod__vix_max_abs_ret_5d", "vix_vs_ma20__div__spx_drawdown_252d"], "is_new": false}, {"model_id": "v2_h3_CALM_XGBoost_N13", "algo": "XGBoost", "regime": "CALM", "horizon": 3, "n_features": 13, "F1_dir": 0.558, "F1_UP_FORT": 0.2947, "F1_DOWN_FORT": 0.3214, "train_start": "2000-11-02", "sampler": "SMOTE", "best_params": "{}", "features": ["WTI_Oil_FRED_ret_5d__minus__vix_max_abs_ret_5d", "NFCI_ret_5d__prod__hmm_p_stress", "VIX_Price_zscore_60d__div__MSTR_Bitcoin3_ret_5d", "AORD_AUS_ret_5d__minus__spx_drawdown_252d", "EWQ_France_zscore_60d", "EWQ_France_ret_20d", "DHR_ret_1d", "MRK_Merck_zscore_60d", "LMT_LockheedMartin_vol_20d", "XLB_Materials_zscore_60d", "QQQ_vol_20d__prod__vix_max_abs_ret_5d", "vix_vs_ma20__div__spx_drawdown_252d", "vix_vs_ma20__minus__YUM_YumBrands_zscore_60d"], "is_new": false}, {"model_id": "v2_h3_CALM_XGBoost_N14", "algo": "XGBoost", "regime": "CALM", "horizon": 3, "n_features": 14, "F1_dir": 0.552, "F1_UP_FORT": 0.3107, "F1_DOWN_FORT": 0.3009, "train_start": "2000-11-02", "sampler": "SMOTE", "best_params": "{}", "features": ["WTI_Oil_FRED_ret_5d__minus__vix_max_abs_ret_5d", "NFCI_ret_5d__prod__hmm_p_stress", "VIX_Price_zscore_60d__div__MSTR_Bitcoin3_ret_5d", "AORD_AUS_ret_5d__minus__spx_drawdown_252d", "EWQ_France_zscore_60d", "EWQ_France_ret_20d", "DHR_ret_1d", "MRK_Merck_zscore_60d", "LMT_LockheedMartin_vol_20d", "XLB_Materials_zscore_60d", "QQQ_vol_20d__prod__vix_max_abs_ret_5d", "vix_vs_ma20__div__spx_drawdown_252d", "vix_vs_ma20__minus__YUM_YumBrands_zscore_60d", "IYM_BasicMaterials_ret_20d"], "is_new": false}, {"model_id": "v2_h3_CALM_XGBoost_N15", "algo": "XGBoost", "regime": "CALM", "horizon": 3, "n_features": 15, "F1_dir": 0.509, "F1_UP_FORT": 0.2558, "F1_DOWN_FORT": 0.3208, "train_start": "2000-11-02", "sampler": "SMOTE", "best_params": "{}", "features": ["WTI_Oil_FRED_ret_5d__minus__vix_max_abs_ret_5d", "NFCI_ret_5d__prod__hmm_p_stress", "VIX_Price_zscore_60d__div__MSTR_Bitcoin3_ret_5d", "AORD_AUS_ret_5d__minus__spx_drawdown_252d", "EWQ_France_zscore_60d", "EWQ_France_ret_20d", "DHR_ret_1d", "MRK_Merck_zscore_60d", "LMT_LockheedMartin_vol_20d", "XLB_Materials_zscore_60d", "QQQ_vol_20d__prod__vix_max_abs_ret_5d", "vix_vs_ma20__div__spx_drawdown_252d", "vix_vs_ma20__minus__YUM_YumBrands_zscore_60d", "IYM_BasicMaterials_ret_20d", "QQQ_vol_20d"], "is_new": false}, {"model_id": "v2_h3_CALM_XGBoost_N16", "algo": "XGBoost", "regime": "CALM", "horizon": 3, "n_features": 16, "F1_dir": 0.5392, "F1_UP_FORT": 0.2529, "F1_DOWN_FORT": 0.3063, "train_start": "2000-11-02", "sampler": "SMOTE", "best_params": "{}", "features": ["WTI_Oil_FRED_ret_5d__minus__vix_max_abs_ret_5d", "NFCI_ret_5d__prod__hmm_p_stress", "VIX_Price_zscore_60d__div__MSTR_Bitcoin3_ret_5d", "AORD_AUS_ret_5d__minus__spx_drawdown_252d", "EWQ_France_zscore_60d", "EWQ_France_ret_20d", "DHR_ret_1d", "MRK_Merck_zscore_60d", "LMT_LockheedMartin_vol_20d", "XLB_Materials_zscore_60d", "QQQ_vol_20d__prod__vix_max_abs_ret_5d", "vix_vs_ma20__div__spx_drawdown_252d", "vix_vs_ma20__minus__YUM_YumBrands_zscore_60d", "IYM_BasicMaterials_ret_20d", "QQQ_vol_20d", "SCHW_Schwab_ret_5d"], "is_new": false}, {"model_id": "v2_h3_CALM_XGBoost_N17", "algo": "XGBoost", "regime": "CALM", "horizon": 3, "n_features": 17, "F1_dir": 0.5647, "F1_UP_FORT": 0.2247, "F1_DOWN_FORT": 0.4118, "train_start": "2000-11-02", "sampler": "SMOTE", "best_params": "{}", "features": ["WTI_Oil_FRED_ret_5d__minus__vix_max_abs_ret_5d", "NFCI_ret_5d__prod__hmm_p_stress", "VIX_Price_zscore_60d__div__MSTR_Bitcoin3_ret_5d", "AORD_AUS_ret_5d__minus__spx_drawdown_252d", "EWQ_France_zscore_60d", "EWQ_France_ret_20d", "DHR_ret_1d", "MRK_Merck_zscore_60d", "LMT_LockheedMartin_vol_20d", "XLB_Materials_zscore_60d", "QQQ_vol_20d__prod__vix_max_abs_ret_5d", "vix_vs_ma20__div__spx_drawdown_252d", "vix_vs_ma20__minus__YUM_YumBrands_zscore_60d", "IYM_BasicMaterials_ret_20d", "QQQ_vol_20d", "SCHW_Schwab_ret_5d", "VIX_Price_zscore_60d__div__vix_max_abs_ret_5d"], "is_new": false}, {"model_id": "v2_h3_CALM_XGBoost_N18", "algo": "XGBoost", "regime": "CALM", "horizon": 3, "n_features": 18, "F1_dir": 0.5136, "F1_UP_FORT": 0.2022, "F1_DOWN_FORT": 0.3654, "train_start": "2000-11-02", "sampler": "SMOTE", "best_params": "{}", "features": ["WTI_Oil_FRED_ret_5d__minus__vix_max_abs_ret_5d", "NFCI_ret_5d__prod__hmm_p_stress", "VIX_Price_zscore_60d__div__MSTR_Bitcoin3_ret_5d", "AORD_AUS_ret_5d__minus__spx_drawdown_252d", "EWQ_France_zscore_60d", "EWQ_France_ret_20d", "DHR_ret_1d", "MRK_Merck_zscore_60d", "LMT_LockheedMartin_vol_20d", "XLB_Materials_zscore_60d", "QQQ_vol_20d__prod__vix_max_abs_ret_5d", "vix_vs_ma20__div__spx_drawdown_252d", "vix_vs_ma20__minus__YUM_YumBrands_zscore_60d", "IYM_BasicMaterials_ret_20d", "QQQ_vol_20d", "SCHW_Schwab_ret_5d", "VIX_Price_zscore_60d__div__vix_max_abs_ret_5d", "EWM_Malaysia_ret_1d"], "is_new": false}, {"model_id": "v2_h3_CALM_XGBoost_N19", "algo": "XGBoost", "regime": "CALM", "horizon": 3, "n_features": 19, "F1_dir": 0.5305, "F1_UP_FORT": 0.1818, "F1_DOWN_FORT": 0.3178, "train_start": "2000-11-02", "sampler": "SMOTE", "best_params": "{}", "features": ["WTI_Oil_FRED_ret_5d__minus__vix_max_abs_ret_5d", "NFCI_ret_5d__prod__hmm_p_stress", "VIX_Price_zscore_60d__div__MSTR_Bitcoin3_ret_5d", "AORD_AUS_ret_5d__minus__spx_drawdown_252d", "EWQ_France_zscore_60d", "EWQ_France_ret_20d", "DHR_ret_1d", "MRK_Merck_zscore_60d", "LMT_LockheedMartin_vol_20d", "XLB_Materials_zscore_60d", "QQQ_vol_20d__prod__vix_max_abs_ret_5d", "vix_vs_ma20__div__spx_drawdown_252d", "vix_vs_ma20__minus__YUM_YumBrands_zscore_60d", "IYM_BasicMaterials_ret_20d", "QQQ_vol_20d", "SCHW_Schwab_ret_5d", "VIX_Price_zscore_60d__div__vix_max_abs_ret_5d", "EWM_Malaysia_ret_1d", "VIX_Price_zscore_60d__div__Russell_Price_ret_5d"], "is_new": false}, {"model_id": "v2_h3_CALM_XGBoost_N22", "algo": "XGBoost", "regime": "CALM", "horizon": 3, "n_features": 22, "F1_dir": 0.5172, "F1_UP_FORT": 0.0988, "F1_DOWN_FORT": 0.3273, "train_start": "2000-11-02", "sampler": "SMOTE", "best_params": "{}", "features": ["WTI_Oil_FRED_ret_5d__minus__vix_max_abs_ret_5d", "NFCI_ret_5d__prod__hmm_p_stress", "VIX_Price_zscore_60d__div__MSTR_Bitcoin3_ret_5d", "AORD_AUS_ret_5d__minus__spx_drawdown_252d", "EWQ_France_zscore_60d", "EWQ_France_ret_20d", "DHR_ret_1d", "MRK_Merck_zscore_60d", "LMT_LockheedMartin_vol_20d", "XLB_Materials_zscore_60d", "QQQ_vol_20d__prod__vix_max_abs_ret_5d", "vix_vs_ma20__div__spx_drawdown_252d", "vix_vs_ma20__minus__YUM_YumBrands_zscore_60d", "IYM_BasicMaterials_ret_20d", "QQQ_vol_20d", "SCHW_Schwab_ret_5d", "VIX_Price_zscore_60d__div__vix_max_abs_ret_5d", "EWM_Malaysia_ret_1d", "VIX_Price_zscore_60d__div__Russell_Price_ret_5d", "VIX_Price_zscore_60d__minus__BTI_BritishAmerican_ret_5d", "QQQ_vol_20d__macross__QQQ_ret_5d", "HangSeng_HK_ret_5d"], "is_new": false}, {"model_id": "v2_h3_CALM_LightGBM_N5", "algo": "LightGBM", "regime": "CALM", "horizon": 3, "n_features": 5, "F1_dir": 0.5362, "F1_UP_FORT": 0.3119, "F1_DOWN_FORT": 0.3636, "train_start": "2000-11-02", "sampler": "SMOTE", "best_params": "{}", "features": ["WTI_Oil_FRED_ret_5d__minus__vix_max_abs_ret_5d", "NFCI_ret_5d__prod__hmm_p_stress", "VIX_Price_zscore_60d__div__MSTR_Bitcoin3_ret_5d", "AORD_AUS_ret_5d__minus__spx_drawdown_252d", "EWQ_France_zscore_60d"], "is_new": false}, {"model_id": "v2_h3_CALM_LightGBM_N6", "algo": "LightGBM", "regime": "CALM", "horizon": 3, "n_features": 6, "F1_dir": 0.5102, "F1_UP_FORT": 0.2316, "F1_DOWN_FORT": 0.2832, "train_start": "2000-11-02", "sampler": "SMOTE", "best_params": "{}", "features": ["WTI_Oil_FRED_ret_5d__minus__vix_max_abs_ret_5d", "NFCI_ret_5d__prod__hmm_p_stress", "VIX_Price_zscore_60d__div__MSTR_Bitcoin3_ret_5d", "AORD_AUS_ret_5d__minus__spx_drawdown_252d", "EWQ_France_zscore_60d", "EWQ_France_ret_20d"], "is_new": false}, {"model_id": "v2_h3_CALM_LightGBM_N7", "algo": "LightGBM", "regime": "CALM", "horizon": 3, "n_features": 7, "F1_dir": 0.5107, "F1_UP_FORT": 0.2391, "F1_DOWN_FORT": 0.2759, "train_start": "2000-11-02", "sampler": "SMOTE", "best_params": "{}", "features": ["WTI_Oil_FRED_ret_5d__minus__vix_max_abs_ret_5d", "NFCI_ret_5d__prod__hmm_p_stress", "VIX_Price_zscore_60d__div__MSTR_Bitcoin3_ret_5d", "AORD_AUS_ret_5d__minus__spx_drawdown_252d", "EWQ_France_zscore_60d", "EWQ_France_ret_20d", "DHR_ret_1d"], "is_new": false}, {"model_id": "v2_h3_CALM_LightGBM_N9", "algo": "LightGBM", "regime": "CALM", "horizon": 3, "n_features": 9, "F1_dir": 0.5003, "F1_UP_FORT": 0.25, "F1_DOWN_FORT": 0.3063, "train_start": "2000-11-02", "sampler": "SMOTE", "best_params": "{}", "features": ["WTI_Oil_FRED_ret_5d__minus__vix_max_abs_ret_5d", "NFCI_ret_5d__prod__hmm_p_stress", "VIX_Price_zscore_60d__div__MSTR_Bitcoin3_ret_5d", "AORD_AUS_ret_5d__minus__spx_drawdown_252d", "EWQ_France_zscore_60d", "EWQ_France_ret_20d", "DHR_ret_1d", "MRK_Merck_zscore_60d", "LMT_LockheedMartin_vol_20d"], "is_new": false}, {"model_id": "v2_h3_CALM_LightGBM_N10", "algo": "LightGBM", "regime": "CALM", "horizon": 3, "n_features": 10, "F1_dir": 0.5584, "F1_UP_FORT": 0.2574, "F1_DOWN_FORT": 0.2975, "train_start": "2000-11-02", "sampler": "SMOTE", "best_params": "{}", "features": ["WTI_Oil_FRED_ret_5d__minus__vix_max_abs_ret_5d", "NFCI_ret_5d__prod__hmm_p_stress", "VIX_Price_zscore_60d__div__MSTR_Bitcoin3_ret_5d", "AORD_AUS_ret_5d__minus__spx_drawdown_252d", "EWQ_France_zscore_60d", "EWQ_France_ret_20d", "DHR_ret_1d", "MRK_Merck_zscore_60d", "LMT_LockheedMartin_vol_20d", "XLB_Materials_zscore_60d"], "is_new": false}, {"model_id": "v2_h3_CALM_LightGBM_N11", "algo": "LightGBM", "regime": "CALM", "horizon": 3, "n_features": 11, "F1_dir": 0.5151, "F1_UP_FORT": 0.1628, "F1_DOWN_FORT": 0.2703, "train_start": "2000-11-02", "sampler": "SMOTE", "best_params": "{}", "features": ["WTI_Oil_FRED_ret_5d__minus__vix_max_abs_ret_5d", "NFCI_ret_5d__prod__hmm_p_stress", "VIX_Price_zscore_60d__div__MSTR_Bitcoin3_ret_5d", "AORD_AUS_ret_5d__minus__spx_drawdown_252d", "EWQ_France_zscore_60d", "EWQ_France_ret_20d", "DHR_ret_1d", "MRK_Merck_zscore_60d", "LMT_LockheedMartin_vol_20d", "XLB_Materials_zscore_60d", "QQQ_vol_20d__prod__vix_max_abs_ret_5d"], "is_new": false}, {"model_id": "v2_h3_CALM_LightGBM_N12", "algo": "LightGBM", "regime": "CALM", "horizon": 3, "n_features": 12, "F1_dir": 0.5186, "F1_UP_FORT": 0.2553, "F1_DOWN_FORT": 0.2703, "train_start": "2000-11-02", "sampler": "SMOTE", "best_params": "{}", "features": ["WTI_Oil_FRED_ret_5d__minus__vix_max_abs_ret_5d", "NFCI_ret_5d__prod__hmm_p_stress", "VIX_Price_zscore_60d__div__MSTR_Bitcoin3_ret_5d", "AORD_AUS_ret_5d__minus__spx_drawdown_252d", "EWQ_France_zscore_60d", "EWQ_France_ret_20d", "DHR_ret_1d", "MRK_Merck_zscore_60d", "LMT_LockheedMartin_vol_20d", "XLB_Materials_zscore_60d", "QQQ_vol_20d__prod__vix_max_abs_ret_5d", "vix_vs_ma20__div__spx_drawdown_252d"], "is_new": false}, {"model_id": "v2_h3_CALM_LightGBM_N13", "algo": "LightGBM", "regime": "CALM", "horizon": 3, "n_features": 13, "F1_dir": 0.5494, "F1_UP_FORT": 0.2766, "F1_DOWN_FORT": 0.3333, "train_start": "2000-11-02", "sampler": "SMOTE", "best_params": "{}", "features": ["WTI_Oil_FRED_ret_5d__minus__vix_max_abs_ret_5d", "NFCI_ret_5d__prod__hmm_p_stress", "VIX_Price_zscore_60d__div__MSTR_Bitcoin3_ret_5d", "AORD_AUS_ret_5d__minus__spx_drawdown_252d", "EWQ_France_zscore_60d", "EWQ_France_ret_20d", "DHR_ret_1d", "MRK_Merck_zscore_60d", "LMT_LockheedMartin_vol_20d", "XLB_Materials_zscore_60d", "QQQ_vol_20d__prod__vix_max_abs_ret_5d", "vix_vs_ma20__div__spx_drawdown_252d", "vix_vs_ma20__minus__YUM_YumBrands_zscore_60d"], "is_new": false}, {"model_id": "v2_h3_CALM_LightGBM_N14", "algo": "LightGBM", "regime": "CALM", "horizon": 3, "n_features": 14, "F1_dir": 0.5273, "F1_UP_FORT": 0.2667, "F1_DOWN_FORT": 0.3248, "train_start": "2000-11-02", "sampler": "SMOTE", "best_params": "{}", "features": ["WTI_Oil_FRED_ret_5d__minus__vix_max_abs_ret_5d", "NFCI_ret_5d__prod__hmm_p_stress", "VIX_Price_zscore_60d__div__MSTR_Bitcoin3_ret_5d", "AORD_AUS_ret_5d__minus__spx_drawdown_252d", "EWQ_France_zscore_60d", "EWQ_France_ret_20d", "DHR_ret_1d", "MRK_Merck_zscore_60d", "LMT_LockheedMartin_vol_20d", "XLB_Materials_zscore_60d", "QQQ_vol_20d__prod__vix_max_abs_ret_5d", "vix_vs_ma20__div__spx_drawdown_252d", "vix_vs_ma20__minus__YUM_YumBrands_zscore_60d", "IYM_BasicMaterials_ret_20d"], "is_new": false}, {"model_id": "v2_h3_CALM_LightGBM_N15", "algo": "LightGBM", "regime": "CALM", "horizon": 3, "n_features": 15, "F1_dir": 0.5223, "F1_UP_FORT": 0.2759, "F1_DOWN_FORT": 0.2991, "train_start": "2000-11-02", "sampler": "SMOTE", "best_params": "{}", "features": ["WTI_Oil_FRED_ret_5d__minus__vix_max_abs_ret_5d", "NFCI_ret_5d__prod__hmm_p_stress", "VIX_Price_zscore_60d__div__MSTR_Bitcoin3_ret_5d", "AORD_AUS_ret_5d__minus__spx_drawdown_252d", "EWQ_France_zscore_60d", "EWQ_France_ret_20d", "DHR_ret_1d", "MRK_Merck_zscore_60d", "LMT_LockheedMartin_vol_20d", "XLB_Materials_zscore_60d", "QQQ_vol_20d__prod__vix_max_abs_ret_5d", "vix_vs_ma20__div__spx_drawdown_252d", "vix_vs_ma20__minus__YUM_YumBrands_zscore_60d", "IYM_BasicMaterials_ret_20d", "QQQ_vol_20d"], "is_new": false}, {"model_id": "v2_h3_CALM_LightGBM_N16", "algo": "LightGBM", "regime": "CALM", "horizon": 3, "n_features": 16, "F1_dir": 0.5355, "F1_UP_FORT": 0.2299, "F1_DOWN_FORT": 0.3455, "train_start": "2000-11-02", "sampler": "SMOTE", "best_params": "{}", "features": ["WTI_Oil_FRED_ret_5d__minus__vix_max_abs_ret_5d", "NFCI_ret_5d__prod__hmm_p_stress", "VIX_Price_zscore_60d__div__MSTR_Bitcoin3_ret_5d", "AORD_AUS_ret_5d__minus__spx_drawdown_252d", "EWQ_France_zscore_60d", "EWQ_France_ret_20d", "DHR_ret_1d", "MRK_Merck_zscore_60d", "LMT_LockheedMartin_vol_20d", "XLB_Materials_zscore_60d", "QQQ_vol_20d__prod__vix_max_abs_ret_5d", "vix_vs_ma20__div__spx_drawdown_252d", "vix_vs_ma20__minus__YUM_YumBrands_zscore_60d", "IYM_BasicMaterials_ret_20d", "QQQ_vol_20d", "SCHW_Schwab_ret_5d"], "is_new": false}, {"model_id": "v2_h3_CALM_LightGBM_N17", "algo": "LightGBM", "regime": "CALM", "horizon": 3, "n_features": 17, "F1_dir": 0.5694, "F1_UP_FORT": 0.1364, "F1_DOWN_FORT": 0.3704, "train_start": "2000-11-02", "sampler": "SMOTE", "best_params": "{}", "features": ["WTI_Oil_FRED_ret_5d__minus__vix_max_abs_ret_5d", "NFCI_ret_5d__prod__hmm_p_stress", "VIX_Price_zscore_60d__div__MSTR_Bitcoin3_ret_5d", "AORD_AUS_ret_5d__minus__spx_drawdown_252d", "EWQ_France_zscore_60d", "EWQ_France_ret_20d", "DHR_ret_1d", "MRK_Merck_zscore_60d", "LMT_LockheedMartin_vol_20d", "XLB_Materials_zscore_60d", "QQQ_vol_20d__prod__vix_max_abs_ret_5d", "vix_vs_ma20__div__spx_drawdown_252d", "vix_vs_ma20__minus__YUM_YumBrands_zscore_60d", "IYM_BasicMaterials_ret_20d", "QQQ_vol_20d", "SCHW_Schwab_ret_5d", "VIX_Price_zscore_60d__div__vix_max_abs_ret_5d"], "is_new": false}, {"model_id": "v2_h3_CALM_LightGBM_N18", "algo": "LightGBM", "regime": "CALM", "horizon": 3, "n_features": 18, "F1_dir": 0.5151, "F1_UP_FORT": 0.2588, "F1_DOWN_FORT": 0.3148, "train_start": "2000-11-02", "sampler": "SMOTE", "best_params": "{}", "features": ["WTI_Oil_FRED_ret_5d__minus__vix_max_abs_ret_5d", "NFCI_ret_5d__prod__hmm_p_stress", "VIX_Price_zscore_60d__div__MSTR_Bitcoin3_ret_5d", "AORD_AUS_ret_5d__minus__spx_drawdown_252d", "EWQ_France_zscore_60d", "EWQ_France_ret_20d", "DHR_ret_1d", "MRK_Merck_zscore_60d", "LMT_LockheedMartin_vol_20d", "XLB_Materials_zscore_60d", "QQQ_vol_20d__prod__vix_max_abs_ret_5d", "vix_vs_ma20__div__spx_drawdown_252d", "vix_vs_ma20__minus__YUM_YumBrands_zscore_60d", "IYM_BasicMaterials_ret_20d", "QQQ_vol_20d", "SCHW_Schwab_ret_5d", "VIX_Price_zscore_60d__div__vix_max_abs_ret_5d", "EWM_Malaysia_ret_1d"], "is_new": false}, {"model_id": "v2_h3_CALM_LightGBM_N19", "algo": "LightGBM", "regime": "CALM", "horizon": 3, "n_features": 19, "F1_dir": 0.5688, "F1_UP_FORT": 0.1882, "F1_DOWN_FORT": 0.3364, "train_start": "2000-11-02", "sampler": "SMOTE", "best_params": "{}", "features": ["WTI_Oil_FRED_ret_5d__minus__vix_max_abs_ret_5d", "NFCI_ret_5d__prod__hmm_p_stress", "VIX_Price_zscore_60d__div__MSTR_Bitcoin3_ret_5d", "AORD_AUS_ret_5d__minus__spx_drawdown_252d", "EWQ_France_zscore_60d", "EWQ_France_ret_20d", "DHR_ret_1d", "MRK_Merck_zscore_60d", "LMT_LockheedMartin_vol_20d", "XLB_Materials_zscore_60d", "QQQ_vol_20d__prod__vix_max_abs_ret_5d", "vix_vs_ma20__div__spx_drawdown_252d", "vix_vs_ma20__minus__YUM_YumBrands_zscore_60d", "IYM_BasicMaterials_ret_20d", "QQQ_vol_20d", "SCHW_Schwab_ret_5d", "VIX_Price_zscore_60d__div__vix_max_abs_ret_5d", "EWM_Malaysia_ret_1d", "VIX_Price_zscore_60d__div__Russell_Price_ret_5d"], "is_new": false}, {"model_id": "v2_h3_CALM_LightGBM_N20", "algo": "LightGBM", "regime": "CALM", "horizon": 3, "n_features": 20, "F1_dir": 0.5123, "F1_UP_FORT": 0.2553, "F1_DOWN_FORT": 0.3093, "train_start": "2000-11-02", "sampler": "SMOTE", "best_params": "{}", "features": ["WTI_Oil_FRED_ret_5d__minus__vix_max_abs_ret_5d", "NFCI_ret_5d__prod__hmm_p_stress", "VIX_Price_zscore_60d__div__MSTR_Bitcoin3_ret_5d", "AORD_AUS_ret_5d__minus__spx_drawdown_252d", "EWQ_France_zscore_60d", "EWQ_France_ret_20d", "DHR_ret_1d", "MRK_Merck_zscore_60d", "LMT_LockheedMartin_vol_20d", "XLB_Materials_zscore_60d", "QQQ_vol_20d__prod__vix_max_abs_ret_5d", "vix_vs_ma20__div__spx_drawdown_252d", "vix_vs_ma20__minus__YUM_YumBrands_zscore_60d", "IYM_BasicMaterials_ret_20d", "QQQ_vol_20d", "SCHW_Schwab_ret_5d", "VIX_Price_zscore_60d__div__vix_max_abs_ret_5d", "EWM_Malaysia_ret_1d", "VIX_Price_zscore_60d__div__Russell_Price_ret_5d", "VIX_Price_zscore_60d__minus__BTI_BritishAmerican_ret_5d"], "is_new": false}, {"model_id": "v2_h3_CALM_LightGBM_N21", "algo": "LightGBM", "regime": "CALM", "horizon": 3, "n_features": 21, "F1_dir": 0.5682, "F1_UP_FORT": 0.2326, "F1_DOWN_FORT": 0.3063, "train_start": "2000-11-02", "sampler": "SMOTE", "best_params": "{}", "features": ["WTI_Oil_FRED_ret_5d__minus__vix_max_abs_ret_5d", "NFCI_ret_5d__prod__hmm_p_stress", "VIX_Price_zscore_60d__div__MSTR_Bitcoin3_ret_5d", "AORD_AUS_ret_5d__minus__spx_drawdown_252d", "EWQ_France_zscore_60d", "EWQ_France_ret_20d", "DHR_ret_1d", "MRK_Merck_zscore_60d", "LMT_LockheedMartin_vol_20d", "XLB_Materials_zscore_60d", "QQQ_vol_20d__prod__vix_max_abs_ret_5d", "vix_vs_ma20__div__spx_drawdown_252d", "vix_vs_ma20__minus__YUM_YumBrands_zscore_60d", "IYM_BasicMaterials_ret_20d", "QQQ_vol_20d", "SCHW_Schwab_ret_5d", "VIX_Price_zscore_60d__div__vix_max_abs_ret_5d", "EWM_Malaysia_ret_1d", "VIX_Price_zscore_60d__div__Russell_Price_ret_5d", "VIX_Price_zscore_60d__minus__BTI_BritishAmerican_ret_5d", "QQQ_vol_20d__macross__QQQ_ret_5d"], "is_new": false}, {"model_id": "v2_h3_CALM_LightGBM_N22", "algo": "LightGBM", "regime": "CALM", "horizon": 3, "n_features": 22, "F1_dir": 0.5317, "F1_UP_FORT": 0.2759, "F1_DOWN_FORT": 0.3366, "train_start": "2000-11-02", "sampler": "SMOTE", "best_params": "{}", "features": ["WTI_Oil_FRED_ret_5d__minus__vix_max_abs_ret_5d", "NFCI_ret_5d__prod__hmm_p_stress", "VIX_Price_zscore_60d__div__MSTR_Bitcoin3_ret_5d", "AORD_AUS_ret_5d__minus__spx_drawdown_252d", "EWQ_France_zscore_60d", "EWQ_France_ret_20d", "DHR_ret_1d", "MRK_Merck_zscore_60d", "LMT_LockheedMartin_vol_20d", "XLB_Materials_zscore_60d", "QQQ_vol_20d__prod__vix_max_abs_ret_5d", "vix_vs_ma20__div__spx_drawdown_252d", "vix_vs_ma20__minus__YUM_YumBrands_zscore_60d", "IYM_BasicMaterials_ret_20d", "QQQ_vol_20d", "SCHW_Schwab_ret_5d", "VIX_Price_zscore_60d__div__vix_max_abs_ret_5d", "EWM_Malaysia_ret_1d", "VIX_Price_zscore_60d__div__Russell_Price_ret_5d", "VIX_Price_zscore_60d__minus__BTI_BritishAmerican_ret_5d", "QQQ_vol_20d__macross__QQQ_ret_5d", "HangSeng_HK_ret_5d"], "is_new": false}, {"model_id": "v2_h3_CALM_GradientBoosting_N5", "algo": "GradientBoosting", "regime": "CALM", "horizon": 3, "n_features": 5, "F1_dir": 0.5365, "F1_UP_FORT": 0.233, "F1_DOWN_FORT": 0.3761, "train_start": "2000-11-02", "sampler": "SMOTE", "best_params": "{}", "features": ["WTI_Oil_FRED_ret_5d__minus__vix_max_abs_ret_5d", "NFCI_ret_5d__prod__hmm_p_stress", "VIX_Price_zscore_60d__div__MSTR_Bitcoin3_ret_5d", "AORD_AUS_ret_5d__minus__spx_drawdown_252d", "EWQ_France_zscore_60d"], "is_new": false}, {"model_id": "v2_h3_CALM_GradientBoosting_N7", "algo": "GradientBoosting", "regime": "CALM", "horizon": 3, "n_features": 7, "F1_dir": 0.5063, "F1_UP_FORT": 0.1798, "F1_DOWN_FORT": 0.314, "train_start": "2000-11-02", "sampler": "SMOTE", "best_params": "{}", "features": ["WTI_Oil_FRED_ret_5d__minus__vix_max_abs_ret_5d", "NFCI_ret_5d__prod__hmm_p_stress", "VIX_Price_zscore_60d__div__MSTR_Bitcoin3_ret_5d", "AORD_AUS_ret_5d__minus__spx_drawdown_252d", "EWQ_France_zscore_60d", "EWQ_France_ret_20d", "DHR_ret_1d"], "is_new": false}, {"model_id": "v2_h3_CALM_GradientBoosting_N8", "algo": "GradientBoosting", "regime": "CALM", "horizon": 3, "n_features": 8, "F1_dir": 0.5063, "F1_UP_FORT": 0.122, "F1_DOWN_FORT": 0.2783, "train_start": "2000-11-02", "sampler": "SMOTE", "best_params": "{}", "features": ["WTI_Oil_FRED_ret_5d__minus__vix_max_abs_ret_5d", "NFCI_ret_5d__prod__hmm_p_stress", "VIX_Price_zscore_60d__div__MSTR_Bitcoin3_ret_5d", "AORD_AUS_ret_5d__minus__spx_drawdown_252d", "EWQ_France_zscore_60d", "EWQ_France_ret_20d", "DHR_ret_1d", "MRK_Merck_zscore_60d"], "is_new": false}, {"model_id": "v2_h3_CALM_GradientBoosting_N9", "algo": "GradientBoosting", "regime": "CALM", "horizon": 3, "n_features": 9, "F1_dir": 0.5016, "F1_UP_FORT": 0.2062, "F1_DOWN_FORT": 0.3, "train_start": "2000-11-02", "sampler": "SMOTE", "best_params": "{}", "features": ["WTI_Oil_FRED_ret_5d__minus__vix_max_abs_ret_5d", "NFCI_ret_5d__prod__hmm_p_stress", "VIX_Price_zscore_60d__div__MSTR_Bitcoin3_ret_5d", "AORD_AUS_ret_5d__minus__spx_drawdown_252d", "EWQ_France_zscore_60d", "EWQ_France_ret_20d", "DHR_ret_1d", "MRK_Merck_zscore_60d", "LMT_LockheedMartin_vol_20d"], "is_new": false}, {"model_id": "v2_h3_CALM_GradientBoosting_N11", "algo": "GradientBoosting", "regime": "CALM", "horizon": 3, "n_features": 11, "F1_dir": 0.5231, "F1_UP_FORT": 0.2857, "F1_DOWN_FORT": 0.2703, "train_start": "2000-11-02", "sampler": "SMOTE", "best_params": "{}", "features": ["WTI_Oil_FRED_ret_5d__minus__vix_max_abs_ret_5d", "NFCI_ret_5d__prod__hmm_p_stress", "VIX_Price_zscore_60d__div__MSTR_Bitcoin3_ret_5d", "AORD_AUS_ret_5d__minus__spx_drawdown_252d", "EWQ_France_zscore_60d", "EWQ_France_ret_20d", "DHR_ret_1d", "MRK_Merck_zscore_60d", "LMT_LockheedMartin_vol_20d", "XLB_Materials_zscore_60d", "QQQ_vol_20d__prod__vix_max_abs_ret_5d"], "is_new": false}, {"model_id": "v2_h3_CALM_GradientBoosting_N12", "algo": "GradientBoosting", "regime": "CALM", "horizon": 3, "n_features": 12, "F1_dir": 0.5438, "F1_UP_FORT": 0.297, "F1_DOWN_FORT": 0.3704, "train_start": "2000-11-02", "sampler": "SMOTE", "best_params": "{}", "features": ["WTI_Oil_FRED_ret_5d__minus__vix_max_abs_ret_5d", "NFCI_ret_5d__prod__hmm_p_stress", "VIX_Price_zscore_60d__div__MSTR_Bitcoin3_ret_5d", "AORD_AUS_ret_5d__minus__spx_drawdown_252d", "EWQ_France_zscore_60d", "EWQ_France_ret_20d", "DHR_ret_1d", "MRK_Merck_zscore_60d", "LMT_LockheedMartin_vol_20d", "XLB_Materials_zscore_60d", "QQQ_vol_20d__prod__vix_max_abs_ret_5d", "vix_vs_ma20__div__spx_drawdown_252d"], "is_new": false}, {"model_id": "v2_h3_CALM_GradientBoosting_N14", "algo": "GradientBoosting", "regime": "CALM", "horizon": 3, "n_features": 14, "F1_dir": 0.5253, "F1_UP_FORT": 0.2737, "F1_DOWN_FORT": 0.3462, "train_start": "2000-11-02", "sampler": "SMOTE", "best_params": "{}", "features": ["WTI_Oil_FRED_ret_5d__minus__vix_max_abs_ret_5d", "NFCI_ret_5d__prod__hmm_p_stress", "VIX_Price_zscore_60d__div__MSTR_Bitcoin3_ret_5d", "AORD_AUS_ret_5d__minus__spx_drawdown_252d", "EWQ_France_zscore_60d", "EWQ_France_ret_20d", "DHR_ret_1d", "MRK_Merck_zscore_60d", "LMT_LockheedMartin_vol_20d", "XLB_Materials_zscore_60d", "QQQ_vol_20d__prod__vix_max_abs_ret_5d", "vix_vs_ma20__div__spx_drawdown_252d", "vix_vs_ma20__minus__YUM_YumBrands_zscore_60d", "IYM_BasicMaterials_ret_20d"], "is_new": false}, {"model_id": "v2_h3_CALM_GradientBoosting_N16", "algo": "GradientBoosting", "regime": "CALM", "horizon": 3, "n_features": 16, "F1_dir": 0.5085, "F1_UP_FORT": 0.2326, "F1_DOWN_FORT": 0.2991, "train_start": "2000-11-02", "sampler": "SMOTE", "best_params": "{}", "features": ["WTI_Oil_FRED_ret_5d__minus__vix_max_abs_ret_5d", "NFCI_ret_5d__prod__hmm_p_stress", "VIX_Price_zscore_60d__div__MSTR_Bitcoin3_ret_5d", "AORD_AUS_ret_5d__minus__spx_drawdown_252d", "EWQ_France_zscore_60d", "EWQ_France_ret_20d", "DHR_ret_1d", "MRK_Merck_zscore_60d", "LMT_LockheedMartin_vol_20d", "XLB_Materials_zscore_60d", "QQQ_vol_20d__prod__vix_max_abs_ret_5d", "vix_vs_ma20__div__spx_drawdown_252d", "vix_vs_ma20__minus__YUM_YumBrands_zscore_60d", "IYM_BasicMaterials_ret_20d", "QQQ_vol_20d", "SCHW_Schwab_ret_5d"], "is_new": false}, {"model_id": "v2_h3_CALM_GradientBoosting_N17", "algo": "GradientBoosting", "regime": "CALM", "horizon": 3, "n_features": 17, "F1_dir": 0.5016, "F1_UP_FORT": 0.2353, "F1_DOWN_FORT": 0.3673, "train_start": "2000-11-02", "sampler": "SMOTE", "best_params": "{}", "features": ["WTI_Oil_FRED_ret_5d__minus__vix_max_abs_ret_5d", "NFCI_ret_5d__prod__hmm_p_stress", "VIX_Price_zscore_60d__div__MSTR_Bitcoin3_ret_5d", "AORD_AUS_ret_5d__minus__spx_drawdown_252d", "EWQ_France_zscore_60d", "EWQ_France_ret_20d", "DHR_ret_1d", "MRK_Merck_zscore_60d", "LMT_LockheedMartin_vol_20d", "XLB_Materials_zscore_60d", "QQQ_vol_20d__prod__vix_max_abs_ret_5d", "vix_vs_ma20__div__spx_drawdown_252d", "vix_vs_ma20__minus__YUM_YumBrands_zscore_60d", "IYM_BasicMaterials_ret_20d", "QQQ_vol_20d", "SCHW_Schwab_ret_5d", "VIX_Price_zscore_60d__div__vix_max_abs_ret_5d"], "is_new": false}, {"model_id": "v2_h3_CALM_GradientBoosting_N18", "algo": "GradientBoosting", "regime": "CALM", "horizon": 3, "n_features": 18, "F1_dir": 0.5293, "F1_UP_FORT": 0.186, "F1_DOWN_FORT": 0.3396, "train_start": "2000-11-02", "sampler": "SMOTE", "best_params": "{}", "features": ["WTI_Oil_FRED_ret_5d__minus__vix_max_abs_ret_5d", "NFCI_ret_5d__prod__hmm_p_stress", "VIX_Price_zscore_60d__div__MSTR_Bitcoin3_ret_5d", "AORD_AUS_ret_5d__minus__spx_drawdown_252d", "EWQ_France_zscore_60d", "EWQ_France_ret_20d", "DHR_ret_1d", "MRK_Merck_zscore_60d", "LMT_LockheedMartin_vol_20d", "XLB_Materials_zscore_60d", "QQQ_vol_20d__prod__vix_max_abs_ret_5d", "vix_vs_ma20__div__spx_drawdown_252d", "vix_vs_ma20__minus__YUM_YumBrands_zscore_60d", "IYM_BasicMaterials_ret_20d", "QQQ_vol_20d", "SCHW_Schwab_ret_5d", "VIX_Price_zscore_60d__div__vix_max_abs_ret_5d", "EWM_Malaysia_ret_1d"], "is_new": false}, {"model_id": "v2_h3_CALM_GradientBoosting_N19", "algo": "GradientBoosting", "regime": "CALM", "horizon": 3, "n_features": 19, "F1_dir": 0.5396, "F1_UP_FORT": 0.2697, "F1_DOWN_FORT": 0.3462, "train_start": "2000-11-02", "sampler": "SMOTE", "best_params": "{}", "features": ["WTI_Oil_FRED_ret_5d__minus__vix_max_abs_ret_5d", "NFCI_ret_5d__prod__hmm_p_stress", "VIX_Price_zscore_60d__div__MSTR_Bitcoin3_ret_5d", "AORD_AUS_ret_5d__minus__spx_drawdown_252d", "EWQ_France_zscore_60d", "EWQ_France_ret_20d", "DHR_ret_1d", "MRK_Merck_zscore_60d", "LMT_LockheedMartin_vol_20d", "XLB_Materials_zscore_60d", "QQQ_vol_20d__prod__vix_max_abs_ret_5d", "vix_vs_ma20__div__spx_drawdown_252d", "vix_vs_ma20__minus__YUM_YumBrands_zscore_60d", "IYM_BasicMaterials_ret_20d", "QQQ_vol_20d", "SCHW_Schwab_ret_5d", "VIX_Price_zscore_60d__div__vix_max_abs_ret_5d", "EWM_Malaysia_ret_1d", "VIX_Price_zscore_60d__div__Russell_Price_ret_5d"], "is_new": false}, {"model_id": "v2_h3_CALM_GradientBoosting_N20", "algo": "GradientBoosting", "regime": "CALM", "horizon": 3, "n_features": 20, "F1_dir": 0.5253, "F1_UP_FORT": 0.2045, "F1_DOWN_FORT": 0.3878, "train_start": "2000-11-02", "sampler": "SMOTE", "best_params": "{}", "features": ["WTI_Oil_FRED_ret_5d__minus__vix_max_abs_ret_5d", "NFCI_ret_5d__prod__hmm_p_stress", "VIX_Price_zscore_60d__div__MSTR_Bitcoin3_ret_5d", "AORD_AUS_ret_5d__minus__spx_drawdown_252d", "EWQ_France_zscore_60d", "EWQ_France_ret_20d", "DHR_ret_1d", "MRK_Merck_zscore_60d", "LMT_LockheedMartin_vol_20d", "XLB_Materials_zscore_60d", "QQQ_vol_20d__prod__vix_max_abs_ret_5d", "vix_vs_ma20__div__spx_drawdown_252d", "vix_vs_ma20__minus__YUM_YumBrands_zscore_60d", "IYM_BasicMaterials_ret_20d", "QQQ_vol_20d", "SCHW_Schwab_ret_5d", "VIX_Price_zscore_60d__div__vix_max_abs_ret_5d", "EWM_Malaysia_ret_1d", "VIX_Price_zscore_60d__div__Russell_Price_ret_5d", "VIX_Price_zscore_60d__minus__BTI_BritishAmerican_ret_5d"], "is_new": false}, {"model_id": "v2_h3_CALM_GradientBoosting_N21", "algo": "GradientBoosting", "regime": "CALM", "horizon": 3, "n_features": 21, "F1_dir": 0.5309, "F1_UP_FORT": 0.2222, "F1_DOWN_FORT": 0.3265, "train_start": "2000-11-02", "sampler": "SMOTE", "best_params": "{}", "features": ["WTI_Oil_FRED_ret_5d__minus__vix_max_abs_ret_5d", "NFCI_ret_5d__prod__hmm_p_stress", "VIX_Price_zscore_60d__div__MSTR_Bitcoin3_ret_5d", "AORD_AUS_ret_5d__minus__spx_drawdown_252d", "EWQ_France_zscore_60d", "EWQ_France_ret_20d", "DHR_ret_1d", "MRK_Merck_zscore_60d", "LMT_LockheedMartin_vol_20d", "XLB_Materials_zscore_60d", "QQQ_vol_20d__prod__vix_max_abs_ret_5d", "vix_vs_ma20__div__spx_drawdown_252d", "vix_vs_ma20__minus__YUM_YumBrands_zscore_60d", "IYM_BasicMaterials_ret_20d", "QQQ_vol_20d", "SCHW_Schwab_ret_5d", "VIX_Price_zscore_60d__div__vix_max_abs_ret_5d", "EWM_Malaysia_ret_1d", "VIX_Price_zscore_60d__div__Russell_Price_ret_5d", "VIX_Price_zscore_60d__minus__BTI_BritishAmerican_ret_5d", "QQQ_vol_20d__macross__QQQ_ret_5d"], "is_new": false}, {"model_id": "v2_h3_CALM_GradientBoosting_N22", "algo": "GradientBoosting", "regime": "CALM", "horizon": 3, "n_features": 22, "F1_dir": 0.5348, "F1_UP_FORT": 0.1667, "F1_DOWN_FORT": 0.3542, "train_start": "2000-11-02", "sampler": "SMOTE", "best_params": "{}", "features": ["WTI_Oil_FRED_ret_5d__minus__vix_max_abs_ret_5d", "NFCI_ret_5d__prod__hmm_p_stress", "VIX_Price_zscore_60d__div__MSTR_Bitcoin3_ret_5d", "AORD_AUS_ret_5d__minus__spx_drawdown_252d", "EWQ_France_zscore_60d", "EWQ_France_ret_20d", "DHR_ret_1d", "MRK_Merck_zscore_60d", "LMT_LockheedMartin_vol_20d", "XLB_Materials_zscore_60d", "QQQ_vol_20d__prod__vix_max_abs_ret_5d", "vix_vs_ma20__div__spx_drawdown_252d", "vix_vs_ma20__minus__YUM_YumBrands_zscore_60d", "IYM_BasicMaterials_ret_20d", "QQQ_vol_20d", "SCHW_Schwab_ret_5d", "VIX_Price_zscore_60d__div__vix_max_abs_ret_5d", "EWM_Malaysia_ret_1d", "VIX_Price_zscore_60d__div__Russell_Price_ret_5d", "VIX_Price_zscore_60d__minus__BTI_BritishAmerican_ret_5d", "QQQ_vol_20d__macross__QQQ_ret_5d", "HangSeng_HK_ret_5d"], "is_new": false}, {"model_id": "v2_h3_CALM_RandomForest_N5", "algo": "RandomForest", "regime": "CALM", "horizon": 3, "n_features": 5, "F1_dir": 0.5538, "F1_UP_FORT": 0.1616, "F1_DOWN_FORT": 0.3871, "train_start": "2000-11-02", "sampler": "SMOTE", "best_params": "{}", "features": ["WTI_Oil_FRED_ret_5d__minus__vix_max_abs_ret_5d", "NFCI_ret_5d__prod__hmm_p_stress", "VIX_Price_zscore_60d__div__MSTR_Bitcoin3_ret_5d", "AORD_AUS_ret_5d__minus__spx_drawdown_252d", "EWQ_France_zscore_60d"], "is_new": false}, {"model_id": "v2_h3_CALM_RandomForest_N6", "algo": "RandomForest", "regime": "CALM", "horizon": 3, "n_features": 6, "F1_dir": 0.5227, "F1_UP_FORT": 0.2316, "F1_DOWN_FORT": 0.3607, "train_start": "2000-11-02", "sampler": "SMOTE", "best_params": "{}", "features": ["WTI_Oil_FRED_ret_5d__minus__vix_max_abs_ret_5d", "NFCI_ret_5d__prod__hmm_p_stress", "VIX_Price_zscore_60d__div__MSTR_Bitcoin3_ret_5d", "AORD_AUS_ret_5d__minus__spx_drawdown_252d", "EWQ_France_zscore_60d", "EWQ_France_ret_20d"], "is_new": false}, {"model_id": "v2_h3_CALM_RandomForest_N7", "algo": "RandomForest", "regime": "CALM", "horizon": 3, "n_features": 7, "F1_dir": 0.5584, "F1_UP_FORT": 0.234, "F1_DOWN_FORT": 0.3692, "train_start": "2000-11-02", "sampler": "SMOTE", "best_params": "{}", "features": ["WTI_Oil_FRED_ret_5d__minus__vix_max_abs_ret_5d", "NFCI_ret_5d__prod__hmm_p_stress", "VIX_Price_zscore_60d__div__MSTR_Bitcoin3_ret_5d", "AORD_AUS_ret_5d__minus__spx_drawdown_252d", "EWQ_France_zscore_60d", "EWQ_France_ret_20d", "DHR_ret_1d"], "is_new": false}, {"model_id": "v2_h3_CALM_RandomForest_N8", "algo": "RandomForest", "regime": "CALM", "horizon": 3, "n_features": 8, "F1_dir": 0.5627, "F1_UP_FORT": 0.2418, "F1_DOWN_FORT": 0.3308, "train_start": "2000-11-02", "sampler": "SMOTE", "best_params": "{}", "features": ["WTI_Oil_FRED_ret_5d__minus__vix_max_abs_ret_5d", "NFCI_ret_5d__prod__hmm_p_stress", "VIX_Price_zscore_60d__div__MSTR_Bitcoin3_ret_5d", "AORD_AUS_ret_5d__minus__spx_drawdown_252d", "EWQ_France_zscore_60d", "EWQ_France_ret_20d", "DHR_ret_1d", "MRK_Merck_zscore_60d"], "is_new": false}, {"model_id": "v2_h3_CALM_RandomForest_N9", "algo": "RandomForest", "regime": "CALM", "horizon": 3, "n_features": 9, "F1_dir": 0.5276, "F1_UP_FORT": 0.22, "F1_DOWN_FORT": 0.2857, "train_start": "2000-11-02", "sampler": "SMOTE", "best_params": "{}", "features": ["WTI_Oil_FRED_ret_5d__minus__vix_max_abs_ret_5d", "NFCI_ret_5d__prod__hmm_p_stress", "VIX_Price_zscore_60d__div__MSTR_Bitcoin3_ret_5d", "AORD_AUS_ret_5d__minus__spx_drawdown_252d", "EWQ_France_zscore_60d", "EWQ_France_ret_20d", "DHR_ret_1d", "MRK_Merck_zscore_60d", "LMT_LockheedMartin_vol_20d"], "is_new": false}, {"model_id": "v2_h3_CALM_RandomForest_N10", "algo": "RandomForest", "regime": "CALM", "horizon": 3, "n_features": 10, "F1_dir": 0.5536, "F1_UP_FORT": 0.2376, "F1_DOWN_FORT": 0.2903, "train_start": "2000-11-02", "sampler": "SMOTE", "best_params": "{}", "features": ["WTI_Oil_FRED_ret_5d__minus__vix_max_abs_ret_5d", "NFCI_ret_5d__prod__hmm_p_stress", "VIX_Price_zscore_60d__div__MSTR_Bitcoin3_ret_5d", "AORD_AUS_ret_5d__minus__spx_drawdown_252d", "EWQ_France_zscore_60d", "EWQ_France_ret_20d", "DHR_ret_1d", "MRK_Merck_zscore_60d", "LMT_LockheedMartin_vol_20d", "XLB_Materials_zscore_60d"], "is_new": false}, {"model_id": "v2_h3_CALM_RandomForest_N11", "algo": "RandomForest", "regime": "CALM", "horizon": 3, "n_features": 11, "F1_dir": 0.567, "F1_UP_FORT": 0.2796, "F1_DOWN_FORT": 0.3307, "train_start": "2000-11-02", "sampler": "SMOTE", "best_params": "{}", "features": ["WTI_Oil_FRED_ret_5d__minus__vix_max_abs_ret_5d", "NFCI_ret_5d__prod__hmm_p_stress", "VIX_Price_zscore_60d__div__MSTR_Bitcoin3_ret_5d", "AORD_AUS_ret_5d__minus__spx_drawdown_252d", "EWQ_France_zscore_60d", "EWQ_France_ret_20d", "DHR_ret_1d", "MRK_Merck_zscore_60d", "LMT_LockheedMartin_vol_20d", "XLB_Materials_zscore_60d", "QQQ_vol_20d__prod__vix_max_abs_ret_5d"], "is_new": false}, {"model_id": "v2_h3_CALM_RandomForest_N12", "algo": "RandomForest", "regime": "CALM", "horizon": 3, "n_features": 12, "F1_dir": 0.5751, "F1_UP_FORT": 0.2553, "F1_DOWN_FORT": 0.3889, "train_start": "2000-11-02", "sampler": "SMOTE", "best_params": "{}", "features": ["WTI_Oil_FRED_ret_5d__minus__vix_max_abs_ret_5d", "NFCI_ret_5d__prod__hmm_p_stress", "VIX_Price_zscore_60d__div__MSTR_Bitcoin3_ret_5d", "AORD_AUS_ret_5d__minus__spx_drawdown_252d", "EWQ_France_zscore_60d", "EWQ_France_ret_20d", "DHR_ret_1d", "MRK_Merck_zscore_60d", "LMT_LockheedMartin_vol_20d", "XLB_Materials_zscore_60d", "QQQ_vol_20d__prod__vix_max_abs_ret_5d", "vix_vs_ma20__div__spx_drawdown_252d"], "is_new": false}, {"model_id": "v2_h3_CALM_RandomForest_N13", "algo": "RandomForest", "regime": "CALM", "horizon": 3, "n_features": 13, "F1_dir": 0.541, "F1_UP_FORT": 0.2222, "F1_DOWN_FORT": 0.3077, "train_start": "2000-11-02", "sampler": "SMOTE", "best_params": "{}", "features": ["WTI_Oil_FRED_ret_5d__minus__vix_max_abs_ret_5d", "NFCI_ret_5d__prod__hmm_p_stress", "VIX_Price_zscore_60d__div__MSTR_Bitcoin3_ret_5d", "AORD_AUS_ret_5d__minus__spx_drawdown_252d", "EWQ_France_zscore_60d", "EWQ_France_ret_20d", "DHR_ret_1d", "MRK_Merck_zscore_60d", "LMT_LockheedMartin_vol_20d", "XLB_Materials_zscore_60d", "QQQ_vol_20d__prod__vix_max_abs_ret_5d", "vix_vs_ma20__div__spx_drawdown_252d", "vix_vs_ma20__minus__YUM_YumBrands_zscore_60d"], "is_new": false}, {"model_id": "v2_h3_CALM_RandomForest_N14", "algo": "RandomForest", "regime": "CALM", "horizon": 3, "n_features": 14, "F1_dir": 0.5404, "F1_UP_FORT": 0.2826, "F1_DOWN_FORT": 0.3214, "train_start": "2000-11-02", "sampler": "SMOTE", "best_params": "{}", "features": ["WTI_Oil_FRED_ret_5d__minus__vix_max_abs_ret_5d", "NFCI_ret_5d__prod__hmm_p_stress", "VIX_Price_zscore_60d__div__MSTR_Bitcoin3_ret_5d", "AORD_AUS_ret_5d__minus__spx_drawdown_252d", "EWQ_France_zscore_60d", "EWQ_France_ret_20d", "DHR_ret_1d", "MRK_Merck_zscore_60d", "LMT_LockheedMartin_vol_20d", "XLB_Materials_zscore_60d", "QQQ_vol_20d__prod__vix_max_abs_ret_5d", "vix_vs_ma20__div__spx_drawdown_252d", "vix_vs_ma20__minus__YUM_YumBrands_zscore_60d", "IYM_BasicMaterials_ret_20d"], "is_new": false}, {"model_id": "v2_h3_CALM_RandomForest_N16", "algo": "RandomForest", "regime": "CALM", "horizon": 3, "n_features": 16, "F1_dir": 0.5105, "F1_UP_FORT": 0.1282, "F1_DOWN_FORT": 0.3455, "train_start": "2000-11-02", "sampler": "SMOTE", "best_params": "{}", "features": ["WTI_Oil_FRED_ret_5d__minus__vix_max_abs_ret_5d", "NFCI_ret_5d__prod__hmm_p_stress", "VIX_Price_zscore_60d__div__MSTR_Bitcoin3_ret_5d", "AORD_AUS_ret_5d__minus__spx_drawdown_252d", "EWQ_France_zscore_60d", "EWQ_France_ret_20d", "DHR_ret_1d", "MRK_Merck_zscore_60d", "LMT_LockheedMartin_vol_20d", "XLB_Materials_zscore_60d", "QQQ_vol_20d__prod__vix_max_abs_ret_5d", "vix_vs_ma20__div__spx_drawdown_252d", "vix_vs_ma20__minus__YUM_YumBrands_zscore_60d", "IYM_BasicMaterials_ret_20d", "QQQ_vol_20d", "SCHW_Schwab_ret_5d"], "is_new": false}, {"model_id": "v2_h3_CALM_RandomForest_N17", "algo": "RandomForest", "regime": "CALM", "horizon": 3, "n_features": 17, "F1_dir": 0.5186, "F1_UP_FORT": 0.1429, "F1_DOWN_FORT": 0.3529, "train_start": "2000-11-02", "sampler": "SMOTE", "best_params": "{}", "features": ["WTI_Oil_FRED_ret_5d__minus__vix_max_abs_ret_5d", "NFCI_ret_5d__prod__hmm_p_stress", "VIX_Price_zscore_60d__div__MSTR_Bitcoin3_ret_5d", "AORD_AUS_ret_5d__minus__spx_drawdown_252d", "EWQ_France_zscore_60d", "EWQ_France_ret_20d", "DHR_ret_1d", "MRK_Merck_zscore_60d", "LMT_LockheedMartin_vol_20d", "XLB_Materials_zscore_60d", "QQQ_vol_20d__prod__vix_max_abs_ret_5d", "vix_vs_ma20__div__spx_drawdown_252d", "vix_vs_ma20__minus__YUM_YumBrands_zscore_60d", "IYM_BasicMaterials_ret_20d", "QQQ_vol_20d", "SCHW_Schwab_ret_5d", "VIX_Price_zscore_60d__div__vix_max_abs_ret_5d"], "is_new": false}, {"model_id": "v2_h3_CALM_RandomForest_N18", "algo": "RandomForest", "regime": "CALM", "horizon": 3, "n_features": 18, "F1_dir": 0.5054, "F1_UP_FORT": 0.1647, "F1_DOWN_FORT": 0.3238, "train_start": "2000-11-02", "sampler": "SMOTE", "best_params": "{}", "features": ["WTI_Oil_FRED_ret_5d__minus__vix_max_abs_ret_5d", "NFCI_ret_5d__prod__hmm_p_stress", "VIX_Price_zscore_60d__div__MSTR_Bitcoin3_ret_5d", "AORD_AUS_ret_5d__minus__spx_drawdown_252d", "EWQ_France_zscore_60d", "EWQ_France_ret_20d", "DHR_ret_1d", "MRK_Merck_zscore_60d", "LMT_LockheedMartin_vol_20d", "XLB_Materials_zscore_60d", "QQQ_vol_20d__prod__vix_max_abs_ret_5d", "vix_vs_ma20__div__spx_drawdown_252d", "vix_vs_ma20__minus__YUM_YumBrands_zscore_60d", "IYM_BasicMaterials_ret_20d", "QQQ_vol_20d", "SCHW_Schwab_ret_5d", "VIX_Price_zscore_60d__div__vix_max_abs_ret_5d", "EWM_Malaysia_ret_1d"], "is_new": false}, {"model_id": "v2_h3_CALM_RandomForest_N20", "algo": "RandomForest", "regime": "CALM", "horizon": 3, "n_features": 20, "F1_dir": 0.5318, "F1_UP_FORT": 0.1818, "F1_DOWN_FORT": 0.3551, "train_start": "2000-11-02", "sampler": "SMOTE", "best_params": "{}", "features": ["WTI_Oil_FRED_ret_5d__minus__vix_max_abs_ret_5d", "NFCI_ret_5d__prod__hmm_p_stress", "VIX_Price_zscore_60d__div__MSTR_Bitcoin3_ret_5d", "AORD_AUS_ret_5d__minus__spx_drawdown_252d", "EWQ_France_zscore_60d", "EWQ_France_ret_20d", "DHR_ret_1d", "MRK_Merck_zscore_60d", "LMT_LockheedMartin_vol_20d", "XLB_Materials_zscore_60d", "QQQ_vol_20d__prod__vix_max_abs_ret_5d", "vix_vs_ma20__div__spx_drawdown_252d", "vix_vs_ma20__minus__YUM_YumBrands_zscore_60d", "IYM_BasicMaterials_ret_20d", "QQQ_vol_20d", "SCHW_Schwab_ret_5d", "VIX_Price_zscore_60d__div__vix_max_abs_ret_5d", "EWM_Malaysia_ret_1d", "VIX_Price_zscore_60d__div__Russell_Price_ret_5d", "VIX_Price_zscore_60d__minus__BTI_BritishAmerican_ret_5d"], "is_new": false}, {"model_id": "v2_h3_CALM_RandomForest_N21", "algo": "RandomForest", "regime": "CALM", "horizon": 3, "n_features": 21, "F1_dir": 0.5314, "F1_UP_FORT": 0.1687, "F1_DOWN_FORT": 0.3519, "train_start": "2000-11-02", "sampler": "SMOTE", "best_params": "{}", "features": ["WTI_Oil_FRED_ret_5d__minus__vix_max_abs_ret_5d", "NFCI_ret_5d__prod__hmm_p_stress", "VIX_Price_zscore_60d__div__MSTR_Bitcoin3_ret_5d", "AORD_AUS_ret_5d__minus__spx_drawdown_252d", "EWQ_France_zscore_60d", "EWQ_France_ret_20d", "DHR_ret_1d", "MRK_Merck_zscore_60d", "LMT_LockheedMartin_vol_20d", "XLB_Materials_zscore_60d", "QQQ_vol_20d__prod__vix_max_abs_ret_5d", "vix_vs_ma20__div__spx_drawdown_252d", "vix_vs_ma20__minus__YUM_YumBrands_zscore_60d", "IYM_BasicMaterials_ret_20d", "QQQ_vol_20d", "SCHW_Schwab_ret_5d", "VIX_Price_zscore_60d__div__vix_max_abs_ret_5d", "EWM_Malaysia_ret_1d", "VIX_Price_zscore_60d__div__Russell_Price_ret_5d", "VIX_Price_zscore_60d__minus__BTI_BritishAmerican_ret_5d", "QQQ_vol_20d__macross__QQQ_ret_5d"], "is_new": false}, {"model_id": "v2_h3_CALM_RandomForest_N22", "algo": "RandomForest", "regime": "CALM", "horizon": 3, "n_features": 22, "F1_dir": 0.5407, "F1_UP_FORT": 0.1647, "F1_DOWN_FORT": 0.3478, "train_start": "2000-11-02", "sampler": "SMOTE", "best_params": "{}", "features": ["WTI_Oil_FRED_ret_5d__minus__vix_max_abs_ret_5d", "NFCI_ret_5d__prod__hmm_p_stress", "VIX_Price_zscore_60d__div__MSTR_Bitcoin3_ret_5d", "AORD_AUS_ret_5d__minus__spx_drawdown_252d", "EWQ_France_zscore_60d", "EWQ_France_ret_20d", "DHR_ret_1d", "MRK_Merck_zscore_60d", "LMT_LockheedMartin_vol_20d", "XLB_Materials_zscore_60d", "QQQ_vol_20d__prod__vix_max_abs_ret_5d", "vix_vs_ma20__div__spx_drawdown_252d", "vix_vs_ma20__minus__YUM_YumBrands_zscore_60d", "IYM_BasicMaterials_ret_20d", "QQQ_vol_20d", "SCHW_Schwab_ret_5d", "VIX_Price_zscore_60d__div__vix_max_abs_ret_5d", "EWM_Malaysia_ret_1d", "VIX_Price_zscore_60d__div__Russell_Price_ret_5d", "VIX_Price_zscore_60d__minus__BTI_BritishAmerican_ret_5d", "QQQ_vol_20d__macross__QQQ_ret_5d", "HangSeng_HK_ret_5d"], "is_new": false}, {"model_id": "v2_h3_CALM_LogisticRegression_N5", "algo": "LogisticRegression", "regime": "CALM", "horizon": 3, "n_features": 5, "F1_dir": 0.5699, "F1_UP_FORT": 0.2804, "F1_DOWN_FORT": 0.3492, "train_start": "2000-11-02", "sampler": "SMOTE", "best_params": "{}", "features": ["WTI_Oil_FRED_ret_5d__minus__vix_max_abs_ret_5d", "NFCI_ret_5d__prod__hmm_p_stress", "VIX_Price_zscore_60d__div__MSTR_Bitcoin3_ret_5d", "AORD_AUS_ret_5d__minus__spx_drawdown_252d", "EWQ_France_zscore_60d"], "is_new": false}, {"model_id": "v2_h3_CALM_LogisticRegression_N6", "algo": "LogisticRegression", "regime": "CALM", "horizon": 3, "n_features": 6, "F1_dir": 0.5707, "F1_UP_FORT": 0.234, "F1_DOWN_FORT": 0.3231, "train_start": "2000-11-02", "sampler": "SMOTE", "best_params": "{}", "features": ["WTI_Oil_FRED_ret_5d__minus__vix_max_abs_ret_5d", "NFCI_ret_5d__prod__hmm_p_stress", "VIX_Price_zscore_60d__div__MSTR_Bitcoin3_ret_5d", "AORD_AUS_ret_5d__minus__spx_drawdown_252d", "EWQ_France_zscore_60d", "EWQ_France_ret_20d"], "is_new": false}, {"model_id": "v2_h3_CALM_LogisticRegression_N7", "algo": "LogisticRegression", "regime": "CALM", "horizon": 3, "n_features": 7, "F1_dir": 0.5611, "F1_UP_FORT": 0.2308, "F1_DOWN_FORT": 0.3167, "train_start": "2000-11-02", "sampler": "SMOTE", "best_params": "{}", "features": ["WTI_Oil_FRED_ret_5d__minus__vix_max_abs_ret_5d", "NFCI_ret_5d__prod__hmm_p_stress", "VIX_Price_zscore_60d__div__MSTR_Bitcoin3_ret_5d", "AORD_AUS_ret_5d__minus__spx_drawdown_252d", "EWQ_France_zscore_60d", "EWQ_France_ret_20d", "DHR_ret_1d"], "is_new": false}, {"model_id": "v2_h3_CALM_LogisticRegression_N8", "algo": "LogisticRegression", "regime": "CALM", "horizon": 3, "n_features": 8, "F1_dir": 0.5755, "F1_UP_FORT": 0.26, "F1_DOWN_FORT": 0.3115, "train_start": "2000-11-02", "sampler": "SMOTE", "best_params": "{}", "features": ["WTI_Oil_FRED_ret_5d__minus__vix_max_abs_ret_5d", "NFCI_ret_5d__prod__hmm_p_stress", "VIX_Price_zscore_60d__div__MSTR_Bitcoin3_ret_5d", "AORD_AUS_ret_5d__minus__spx_drawdown_252d", "EWQ_France_zscore_60d", "EWQ_France_ret_20d", "DHR_ret_1d", "MRK_Merck_zscore_60d"], "is_new": false}, {"model_id": "v2_h3_CALM_LogisticRegression_N9", "algo": "LogisticRegression", "regime": "CALM", "horizon": 3, "n_features": 9, "F1_dir": 0.5467, "F1_UP_FORT": 0.3243, "F1_DOWN_FORT": 0.2881, "train_start": "2000-11-02", "sampler": "SMOTE", "best_params": "{}", "features": ["WTI_Oil_FRED_ret_5d__minus__vix_max_abs_ret_5d", "NFCI_ret_5d__prod__hmm_p_stress", "VIX_Price_zscore_60d__div__MSTR_Bitcoin3_ret_5d", "AORD_AUS_ret_5d__minus__spx_drawdown_252d", "EWQ_France_zscore_60d", "EWQ_France_ret_20d", "DHR_ret_1d", "MRK_Merck_zscore_60d", "LMT_LockheedMartin_vol_20d"], "is_new": false}, {"model_id": "v2_h3_CALM_LogisticRegression_N10", "algo": "LogisticRegression", "regime": "CALM", "horizon": 3, "n_features": 10, "F1_dir": 0.5548, "F1_UP_FORT": 0.2632, "F1_DOWN_FORT": 0.2857, "train_start": "2000-11-02", "sampler": "SMOTE", "best_params": "{}", "features": ["WTI_Oil_FRED_ret_5d__minus__vix_max_abs_ret_5d", "NFCI_ret_5d__prod__hmm_p_stress", "VIX_Price_zscore_60d__div__MSTR_Bitcoin3_ret_5d", "AORD_AUS_ret_5d__minus__spx_drawdown_252d", "EWQ_France_zscore_60d", "EWQ_France_ret_20d", "DHR_ret_1d", "MRK_Merck_zscore_60d", "LMT_LockheedMartin_vol_20d", "XLB_Materials_zscore_60d"], "is_new": false}, {"model_id": "v2_h3_CALM_LogisticRegression_N11", "algo": "LogisticRegression", "regime": "CALM", "horizon": 3, "n_features": 11, "F1_dir": 0.5548, "F1_UP_FORT": 0.2549, "F1_DOWN_FORT": 0.3063, "train_start": "2000-11-02", "sampler": "SMOTE", "best_params": "{}", "features": ["WTI_Oil_FRED_ret_5d__minus__vix_max_abs_ret_5d", "NFCI_ret_5d__prod__hmm_p_stress", "VIX_Price_zscore_60d__div__MSTR_Bitcoin3_ret_5d", "AORD_AUS_ret_5d__minus__spx_drawdown_252d", "EWQ_France_zscore_60d", "EWQ_France_ret_20d", "DHR_ret_1d", "MRK_Merck_zscore_60d", "LMT_LockheedMartin_vol_20d", "XLB_Materials_zscore_60d", "QQQ_vol_20d__prod__vix_max_abs_ret_5d"], "is_new": false}, {"model_id": "v2_h3_CALM_LogisticRegression_N12", "algo": "LogisticRegression", "regime": "CALM", "horizon": 3, "n_features": 12, "F1_dir": 0.5682, "F1_UP_FORT": 0.26, "F1_DOWN_FORT": 0.2936, "train_start": "2000-11-02", "sampler": "SMOTE", "best_params": "{}", "features": ["WTI_Oil_FRED_ret_5d__minus__vix_max_abs_ret_5d", "NFCI_ret_5d__prod__hmm_p_stress", "VIX_Price_zscore_60d__div__MSTR_Bitcoin3_ret_5d", "AORD_AUS_ret_5d__minus__spx_drawdown_252d", "EWQ_France_zscore_60d", "EWQ_France_ret_20d", "DHR_ret_1d", "MRK_Merck_zscore_60d", "LMT_LockheedMartin_vol_20d", "XLB_Materials_zscore_60d", "QQQ_vol_20d__prod__vix_max_abs_ret_5d", "vix_vs_ma20__div__spx_drawdown_252d"], "is_new": false}, {"model_id": "v2_h3_CALM_LogisticRegression_N13", "algo": "LogisticRegression", "regime": "CALM", "horizon": 3, "n_features": 13, "F1_dir": 0.5622, "F1_UP_FORT": 0.2069, "F1_DOWN_FORT": 0.3167, "train_start": "2000-11-02", "sampler": "SMOTE", "best_params": "{}", "features": ["WTI_Oil_FRED_ret_5d__minus__vix_max_abs_ret_5d", "NFCI_ret_5d__prod__hmm_p_stress", "VIX_Price_zscore_60d__div__MSTR_Bitcoin3_ret_5d", "AORD_AUS_ret_5d__minus__spx_drawdown_252d", "EWQ_France_zscore_60d", "EWQ_France_ret_20d", "DHR_ret_1d", "MRK_Merck_zscore_60d", "LMT_LockheedMartin_vol_20d", "XLB_Materials_zscore_60d", "QQQ_vol_20d__prod__vix_max_abs_ret_5d", "vix_vs_ma20__div__spx_drawdown_252d", "vix_vs_ma20__minus__YUM_YumBrands_zscore_60d"], "is_new": false}, {"model_id": "v2_h3_CALM_LogisticRegression_N14", "algo": "LogisticRegression", "regime": "CALM", "horizon": 3, "n_features": 14, "F1_dir": 0.5661, "F1_UP_FORT": 0.2022, "F1_DOWN_FORT": 0.3077, "train_start": "2000-11-02", "sampler": "SMOTE", "best_params": "{}", "features": ["WTI_Oil_FRED_ret_5d__minus__vix_max_abs_ret_5d", "NFCI_ret_5d__prod__hmm_p_stress", "VIX_Price_zscore_60d__div__MSTR_Bitcoin3_ret_5d", "AORD_AUS_ret_5d__minus__spx_drawdown_252d", "EWQ_France_zscore_60d", "EWQ_France_ret_20d", "DHR_ret_1d", "MRK_Merck_zscore_60d", "LMT_LockheedMartin_vol_20d", "XLB_Materials_zscore_60d", "QQQ_vol_20d__prod__vix_max_abs_ret_5d", "vix_vs_ma20__div__spx_drawdown_252d", "vix_vs_ma20__minus__YUM_YumBrands_zscore_60d", "IYM_BasicMaterials_ret_20d"], "is_new": false}, {"model_id": "v2_h3_CALM_LogisticRegression_N15", "algo": "LogisticRegression", "regime": "CALM", "horizon": 3, "n_features": 15, "F1_dir": 0.5322, "F1_UP_FORT": 0.0857, "F1_DOWN_FORT": 0.3387, "train_start": "2000-11-02", "sampler": "SMOTE", "best_params": "{}", "features": ["WTI_Oil_FRED_ret_5d__minus__vix_max_abs_ret_5d", "NFCI_ret_5d__prod__hmm_p_stress", "VIX_Price_zscore_60d__div__MSTR_Bitcoin3_ret_5d", "AORD_AUS_ret_5d__minus__spx_drawdown_252d", "EWQ_France_zscore_60d", "EWQ_France_ret_20d", "DHR_ret_1d", "MRK_Merck_zscore_60d", "LMT_LockheedMartin_vol_20d", "XLB_Materials_zscore_60d", "QQQ_vol_20d__prod__vix_max_abs_ret_5d", "vix_vs_ma20__div__spx_drawdown_252d", "vix_vs_ma20__minus__YUM_YumBrands_zscore_60d", "IYM_BasicMaterials_ret_20d", "QQQ_vol_20d"], "is_new": false}, {"model_id": "v2_h3_CALM_LogisticRegression_N16", "algo": "LogisticRegression", "regime": "CALM", "horizon": 3, "n_features": 16, "F1_dir": 0.5278, "F1_UP_FORT": 0.169, "F1_DOWN_FORT": 0.3175, "train_start": "2000-11-02", "sampler": "SMOTE", "best_params": "{}", "features": ["WTI_Oil_FRED_ret_5d__minus__vix_max_abs_ret_5d", "NFCI_ret_5d__prod__hmm_p_stress", "VIX_Price_zscore_60d__div__MSTR_Bitcoin3_ret_5d", "AORD_AUS_ret_5d__minus__spx_drawdown_252d", "EWQ_France_zscore_60d", "EWQ_France_ret_20d", "DHR_ret_1d", "MRK_Merck_zscore_60d", "LMT_LockheedMartin_vol_20d", "XLB_Materials_zscore_60d", "QQQ_vol_20d__prod__vix_max_abs_ret_5d", "vix_vs_ma20__div__spx_drawdown_252d", "vix_vs_ma20__minus__YUM_YumBrands_zscore_60d", "IYM_BasicMaterials_ret_20d", "QQQ_vol_20d", "SCHW_Schwab_ret_5d"], "is_new": false}, {"model_id": "v2_h3_CALM_LogisticRegression_N17", "algo": "LogisticRegression", "regime": "CALM", "horizon": 3, "n_features": 17, "F1_dir": 0.5365, "F1_UP_FORT": 0.2133, "F1_DOWN_FORT": 0.3607, "train_start": "2000-11-02", "sampler": "SMOTE", "best_params": "{}", "features": ["WTI_Oil_FRED_ret_5d__minus__vix_max_abs_ret_5d", "NFCI_ret_5d__prod__hmm_p_stress", "VIX_Price_zscore_60d__div__MSTR_Bitcoin3_ret_5d", "AORD_AUS_ret_5d__minus__spx_drawdown_252d", "EWQ_France_zscore_60d", "EWQ_France_ret_20d", "DHR_ret_1d", "MRK_Merck_zscore_60d", "LMT_LockheedMartin_vol_20d", "XLB_Materials_zscore_60d", "QQQ_vol_20d__prod__vix_max_abs_ret_5d", "vix_vs_ma20__div__spx_drawdown_252d", "vix_vs_ma20__minus__YUM_YumBrands_zscore_60d", "IYM_BasicMaterials_ret_20d", "QQQ_vol_20d", "SCHW_Schwab_ret_5d", "VIX_Price_zscore_60d__div__vix_max_abs_ret_5d"], "is_new": false}, {"model_id": "v2_h3_CALM_LogisticRegression_N18", "algo": "LogisticRegression", "regime": "CALM", "horizon": 3, "n_features": 18, "F1_dir": 0.5404, "F1_UP_FORT": 0.2078, "F1_DOWN_FORT": 0.35, "train_start": "2000-11-02", "sampler": "SMOTE", "best_params": "{}", "features": ["WTI_Oil_FRED_ret_5d__minus__vix_max_abs_ret_5d", "NFCI_ret_5d__prod__hmm_p_stress", "VIX_Price_zscore_60d__div__MSTR_Bitcoin3_ret_5d", "AORD_AUS_ret_5d__minus__spx_drawdown_252d", "EWQ_France_zscore_60d", "EWQ_France_ret_20d", "DHR_ret_1d", "MRK_Merck_zscore_60d", "LMT_LockheedMartin_vol_20d", "XLB_Materials_zscore_60d", "QQQ_vol_20d__prod__vix_max_abs_ret_5d", "vix_vs_ma20__div__spx_drawdown_252d", "vix_vs_ma20__minus__YUM_YumBrands_zscore_60d", "IYM_BasicMaterials_ret_20d", "QQQ_vol_20d", "SCHW_Schwab_ret_5d", "VIX_Price_zscore_60d__div__vix_max_abs_ret_5d", "EWM_Malaysia_ret_1d"], "is_new": false}, {"model_id": "v2_h3_CALM_LogisticRegression_N19", "algo": "LogisticRegression", "regime": "CALM", "horizon": 3, "n_features": 19, "F1_dir": 0.5442, "F1_UP_FORT": 0.16, "F1_DOWN_FORT": 0.3248, "train_start": "2000-11-02", "sampler": "SMOTE", "best_params": "{}", "features": ["WTI_Oil_FRED_ret_5d__minus__vix_max_abs_ret_5d", "NFCI_ret_5d__prod__hmm_p_stress", "VIX_Price_zscore_60d__div__MSTR_Bitcoin3_ret_5d", "AORD_AUS_ret_5d__minus__spx_drawdown_252d", "EWQ_France_zscore_60d", "EWQ_France_ret_20d", "DHR_ret_1d", "MRK_Merck_zscore_60d", "LMT_LockheedMartin_vol_20d", "XLB_Materials_zscore_60d", "QQQ_vol_20d__prod__vix_max_abs_ret_5d", "vix_vs_ma20__div__spx_drawdown_252d", "vix_vs_ma20__minus__YUM_YumBrands_zscore_60d", "IYM_BasicMaterials_ret_20d", "QQQ_vol_20d", "SCHW_Schwab_ret_5d", "VIX_Price_zscore_60d__div__vix_max_abs_ret_5d", "EWM_Malaysia_ret_1d", "VIX_Price_zscore_60d__div__Russell_Price_ret_5d"], "is_new": false}, {"model_id": "v2_h3_CALM_LogisticRegression_N20", "algo": "LogisticRegression", "regime": "CALM", "horizon": 3, "n_features": 20, "F1_dir": 0.579, "F1_UP_FORT": 0.1818, "F1_DOWN_FORT": 0.3419, "train_start": "2000-11-02", "sampler": "SMOTE", "best_params": "{}", "features": ["WTI_Oil_FRED_ret_5d__minus__vix_max_abs_ret_5d", "NFCI_ret_5d__prod__hmm_p_stress", "VIX_Price_zscore_60d__div__MSTR_Bitcoin3_ret_5d", "AORD_AUS_ret_5d__minus__spx_drawdown_252d", "EWQ_France_zscore_60d", "EWQ_France_ret_20d", "DHR_ret_1d", "MRK_Merck_zscore_60d", "LMT_LockheedMartin_vol_20d", "XLB_Materials_zscore_60d", "QQQ_vol_20d__prod__vix_max_abs_ret_5d", "vix_vs_ma20__div__spx_drawdown_252d", "vix_vs_ma20__minus__YUM_YumBrands_zscore_60d", "IYM_BasicMaterials_ret_20d", "QQQ_vol_20d", "SCHW_Schwab_ret_5d", "VIX_Price_zscore_60d__div__vix_max_abs_ret_5d", "EWM_Malaysia_ret_1d", "VIX_Price_zscore_60d__div__Russell_Price_ret_5d", "VIX_Price_zscore_60d__minus__BTI_BritishAmerican_ret_5d"], "is_new": false}, {"model_id": "v2_h3_CALM_LogisticRegression_N21", "algo": "LogisticRegression", "regime": "CALM", "horizon": 3, "n_features": 21, "F1_dir": 0.5664, "F1_UP_FORT": 0.16, "F1_DOWN_FORT": 0.339, "train_start": "2000-11-02", "sampler": "SMOTE", "best_params": "{}", "features": ["WTI_Oil_FRED_ret_5d__minus__vix_max_abs_ret_5d", "NFCI_ret_5d__prod__hmm_p_stress", "VIX_Price_zscore_60d__div__MSTR_Bitcoin3_ret_5d", "AORD_AUS_ret_5d__minus__spx_drawdown_252d", "EWQ_France_zscore_60d", "EWQ_France_ret_20d", "DHR_ret_1d", "MRK_Merck_zscore_60d", "LMT_LockheedMartin_vol_20d", "XLB_Materials_zscore_60d", "QQQ_vol_20d__prod__vix_max_abs_ret_5d", "vix_vs_ma20__div__spx_drawdown_252d", "vix_vs_ma20__minus__YUM_YumBrands_zscore_60d", "IYM_BasicMaterials_ret_20d", "QQQ_vol_20d", "SCHW_Schwab_ret_5d", "VIX_Price_zscore_60d__div__vix_max_abs_ret_5d", "EWM_Malaysia_ret_1d", "VIX_Price_zscore_60d__div__Russell_Price_ret_5d", "VIX_Price_zscore_60d__minus__BTI_BritishAmerican_ret_5d", "QQQ_vol_20d__macross__QQQ_ret_5d"], "is_new": false}, {"model_id": "v2_h3_CALM_LogisticRegression_N22", "algo": "LogisticRegression", "regime": "CALM", "horizon": 3, "n_features": 22, "F1_dir": 0.5533, "F1_UP_FORT": 0.1351, "F1_DOWN_FORT": 0.3559, "train_start": "2000-11-02", "sampler": "SMOTE", "best_params": "{}", "features": ["WTI_Oil_FRED_ret_5d__minus__vix_max_abs_ret_5d", "NFCI_ret_5d__prod__hmm_p_stress", "VIX_Price_zscore_60d__div__MSTR_Bitcoin3_ret_5d", "AORD_AUS_ret_5d__minus__spx_drawdown_252d", "EWQ_France_zscore_60d", "EWQ_France_ret_20d", "DHR_ret_1d", "MRK_Merck_zscore_60d", "LMT_LockheedMartin_vol_20d", "XLB_Materials_zscore_60d", "QQQ_vol_20d__prod__vix_max_abs_ret_5d", "vix_vs_ma20__div__spx_drawdown_252d", "vix_vs_ma20__minus__YUM_YumBrands_zscore_60d", "IYM_BasicMaterials_ret_20d", "QQQ_vol_20d", "SCHW_Schwab_ret_5d", "VIX_Price_zscore_60d__div__vix_max_abs_ret_5d", "EWM_Malaysia_ret_1d", "VIX_Price_zscore_60d__div__Russell_Price_ret_5d", "VIX_Price_zscore_60d__minus__BTI_BritishAmerican_ret_5d", "QQQ_vol_20d__macross__QQQ_ret_5d", "HangSeng_HK_ret_5d"], "is_new": false}, {"model_id": "v2_h3_CALM_XGBoost_Optuna_N17", "algo": "XGBoost", "regime": "CALM", "horizon": 3, "n_features": 17, "F1_dir": 0.558, "F1_UP_FORT": 0.1739, "F1_DOWN_FORT": 0.404, "train_start": "2000-11-02", "sampler": "SMOTE", "best_params": "{}", "features": ["WTI_Oil_FRED_ret_5d__minus__vix_max_abs_ret_5d", "NFCI_ret_5d__prod__hmm_p_stress", "VIX_Price_zscore_60d__div__MSTR_Bitcoin3_ret_5d", "AORD_AUS_ret_5d__minus__spx_drawdown_252d", "EWQ_France_zscore_60d", "EWQ_France_ret_20d", "DHR_ret_1d", "MRK_Merck_zscore_60d", "LMT_LockheedMartin_vol_20d", "XLB_Materials_zscore_60d", "QQQ_vol_20d__prod__vix_max_abs_ret_5d", "vix_vs_ma20__div__spx_drawdown_252d", "vix_vs_ma20__minus__YUM_YumBrands_zscore_60d", "IYM_BasicMaterials_ret_20d", "QQQ_vol_20d", "SCHW_Schwab_ret_5d", "VIX_Price_zscore_60d__div__vix_max_abs_ret_5d"], "is_new": false}, {"model_id": "v2_h3_CALM_XGBoost_OptunaCal_N17", "algo": "XGBoostCal", "regime": "CALM", "horizon": 3, "n_features": 17, "F1_dir": 0.5507, "F1_UP_FORT": 0.1591, "F1_DOWN_FORT": 0.3958, "train_start": "2000-11-02", "sampler": "SMOTE", "best_params": "{}", "features": ["WTI_Oil_FRED_ret_5d__minus__vix_max_abs_ret_5d", "NFCI_ret_5d__prod__hmm_p_stress", "VIX_Price_zscore_60d__div__MSTR_Bitcoin3_ret_5d", "AORD_AUS_ret_5d__minus__spx_drawdown_252d", "EWQ_France_zscore_60d", "EWQ_France_ret_20d", "DHR_ret_1d", "MRK_Merck_zscore_60d", "LMT_LockheedMartin_vol_20d", "XLB_Materials_zscore_60d", "QQQ_vol_20d__prod__vix_max_abs_ret_5d", "vix_vs_ma20__div__spx_drawdown_252d", "vix_vs_ma20__minus__YUM_YumBrands_zscore_60d", "IYM_BasicMaterials_ret_20d", "QQQ_vol_20d", "SCHW_Schwab_ret_5d", "VIX_Price_zscore_60d__div__vix_max_abs_ret_5d"], "is_new": false}, {"model_id": "v2_h3_NORMAL_XGBoost_N5", "algo": "XGBoost", "regime": "NORMAL", "horizon": 3, "n_features": 5, "F1_dir": 0.5366, "F1_UP_FORT": 0.3437, "F1_DOWN_FORT": 0.3436, "train_start": "2001-02-20", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VIX_Price_zscore_60d__div__hmm_p_stress", "vix_zscore_10d__div__heston_var_ev_h1", "NFCI_ret_5d__zrel__NEE_NextEra_ret_5d", "VIX_Price_zscore_60d__zrel__heston_var_ev_h3", "HangSeng_HK_ret_5d"], "is_new": false}, {"model_id": "v2_h3_NORMAL_XGBoost_N6", "algo": "XGBoost", "regime": "NORMAL", "horizon": 3, "n_features": 6, "F1_dir": 0.5545, "F1_UP_FORT": 0.356, "F1_DOWN_FORT": 0.3444, "train_start": "2001-02-20", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VIX_Price_zscore_60d__div__hmm_p_stress", "vix_zscore_10d__div__heston_var_ev_h1", "NFCI_ret_5d__zrel__NEE_NextEra_ret_5d", "VIX_Price_zscore_60d__zrel__heston_var_ev_h3", "HangSeng_HK_ret_5d", "VIX_Price_zscore_60d__ret5x__COF_CapitalOne_ret_5d"], "is_new": false}, {"model_id": "v2_h3_NORMAL_XGBoost_N7", "algo": "XGBoost", "regime": "NORMAL", "horizon": 3, "n_features": 7, "F1_dir": 0.556, "F1_UP_FORT": 0.3526, "F1_DOWN_FORT": 0.3745, "train_start": "2001-02-20", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VIX_Price_zscore_60d__div__hmm_p_stress", "vix_zscore_10d__div__heston_var_ev_h1", "NFCI_ret_5d__zrel__NEE_NextEra_ret_5d", "VIX_Price_zscore_60d__zrel__heston_var_ev_h3", "HangSeng_HK_ret_5d", "VIX_Price_zscore_60d__ret5x__COF_CapitalOne_ret_5d", "DUK_Duke_ret_5d__div__COF_CapitalOne_ret_5d"], "is_new": false}, {"model_id": "v2_h3_NORMAL_XGBoost_N8", "algo": "XGBoost", "regime": "NORMAL", "horizon": 3, "n_features": 8, "F1_dir": 0.5311, "F1_UP_FORT": 0.3325, "F1_DOWN_FORT": 0.3318, "train_start": "2001-02-20", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VIX_Price_zscore_60d__div__hmm_p_stress", "vix_zscore_10d__div__heston_var_ev_h1", "NFCI_ret_5d__zrel__NEE_NextEra_ret_5d", "VIX_Price_zscore_60d__zrel__heston_var_ev_h3", "HangSeng_HK_ret_5d", "VIX_Price_zscore_60d__ret5x__COF_CapitalOne_ret_5d", "DUK_Duke_ret_5d__div__COF_CapitalOne_ret_5d", "DUK_Duke_ret_5d__ret5x__COF_CapitalOne_ret_5d"], "is_new": false}, {"model_id": "v2_h3_NORMAL_XGBoost_N9", "algo": "XGBoost", "regime": "NORMAL", "horizon": 3, "n_features": 9, "F1_dir": 0.5449, "F1_UP_FORT": 0.358, "F1_DOWN_FORT": 0.3426, "train_start": "2001-02-20", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VIX_Price_zscore_60d__div__hmm_p_stress", "vix_zscore_10d__div__heston_var_ev_h1", "NFCI_ret_5d__zrel__NEE_NextEra_ret_5d", "VIX_Price_zscore_60d__zrel__heston_var_ev_h3", "HangSeng_HK_ret_5d", "VIX_Price_zscore_60d__ret5x__COF_CapitalOne_ret_5d", "DUK_Duke_ret_5d__div__COF_CapitalOne_ret_5d", "DUK_Duke_ret_5d__ret5x__COF_CapitalOne_ret_5d", "EWL_Switzerland_vol_20d"], "is_new": false}, {"model_id": "v2_h3_NORMAL_XGBoost_N10", "algo": "XGBoost", "regime": "NORMAL", "horizon": 3, "n_features": 10, "F1_dir": 0.5062, "F1_UP_FORT": 0.3227, "F1_DOWN_FORT": 0.307, "train_start": "2001-02-20", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VIX_Price_zscore_60d__div__hmm_p_stress", "vix_zscore_10d__div__heston_var_ev_h1", "NFCI_ret_5d__zrel__NEE_NextEra_ret_5d", "VIX_Price_zscore_60d__zrel__heston_var_ev_h3", "HangSeng_HK_ret_5d", "VIX_Price_zscore_60d__ret5x__COF_CapitalOne_ret_5d", "DUK_Duke_ret_5d__div__COF_CapitalOne_ret_5d", "DUK_Duke_ret_5d__ret5x__COF_CapitalOne_ret_5d", "EWL_Switzerland_vol_20d", "EWM_Malaysia_vol_20d"], "is_new": false}, {"model_id": "v2_h3_NORMAL_XGBoost_N11", "algo": "XGBoost", "regime": "NORMAL", "horizon": 3, "n_features": 11, "F1_dir": 0.5529, "F1_UP_FORT": 0.3431, "F1_DOWN_FORT": 0.343, "train_start": "2001-02-20", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VIX_Price_zscore_60d__div__hmm_p_stress", "vix_zscore_10d__div__heston_var_ev_h1", "NFCI_ret_5d__zrel__NEE_NextEra_ret_5d", "VIX_Price_zscore_60d__zrel__heston_var_ev_h3", "HangSeng_HK_ret_5d", "VIX_Price_zscore_60d__ret5x__COF_CapitalOne_ret_5d", "DUK_Duke_ret_5d__div__COF_CapitalOne_ret_5d", "DUK_Duke_ret_5d__ret5x__COF_CapitalOne_ret_5d", "EWL_Switzerland_vol_20d", "EWM_Malaysia_vol_20d", "EOG_EOGResources_vol_20d"], "is_new": false}, {"model_id": "v2_h3_NORMAL_XGBoost_N12", "algo": "XGBoost", "regime": "NORMAL", "horizon": 3, "n_features": 12, "F1_dir": 0.5828, "F1_UP_FORT": 0.3713, "F1_DOWN_FORT": 0.3606, "train_start": "2001-02-20", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VIX_Price_zscore_60d__div__hmm_p_stress", "vix_zscore_10d__div__heston_var_ev_h1", "NFCI_ret_5d__zrel__NEE_NextEra_ret_5d", "VIX_Price_zscore_60d__zrel__heston_var_ev_h3", "HangSeng_HK_ret_5d", "VIX_Price_zscore_60d__ret5x__COF_CapitalOne_ret_5d", "DUK_Duke_ret_5d__div__COF_CapitalOne_ret_5d", "DUK_Duke_ret_5d__ret5x__COF_CapitalOne_ret_5d", "EWL_Switzerland_vol_20d", "EWM_Malaysia_vol_20d", "EOG_EOGResources_vol_20d", "kalman_filtered__prod__VOD_Vodafone_vol_20d"], "is_new": false}, {"model_id": "v2_h3_NORMAL_XGBoost_N13", "algo": "XGBoost", "regime": "NORMAL", "horizon": 3, "n_features": 13, "F1_dir": 0.567, "F1_UP_FORT": 0.3614, "F1_DOWN_FORT": 0.3654, "train_start": "2001-02-20", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VIX_Price_zscore_60d__div__hmm_p_stress", "vix_zscore_10d__div__heston_var_ev_h1", "NFCI_ret_5d__zrel__NEE_NextEra_ret_5d", "VIX_Price_zscore_60d__zrel__heston_var_ev_h3", "HangSeng_HK_ret_5d", "VIX_Price_zscore_60d__ret5x__COF_CapitalOne_ret_5d", "DUK_Duke_ret_5d__div__COF_CapitalOne_ret_5d", "DUK_Duke_ret_5d__ret5x__COF_CapitalOne_ret_5d", "EWL_Switzerland_vol_20d", "EWM_Malaysia_vol_20d", "EOG_EOGResources_vol_20d", "kalman_filtered__prod__VOD_Vodafone_vol_20d", "MSFT_ret_5d__div__NEE_NextEra_ret_5d"], "is_new": false}, {"model_id": "v2_h3_NORMAL_XGBoost_N14", "algo": "XGBoost", "regime": "NORMAL", "horizon": 3, "n_features": 14, "F1_dir": 0.595, "F1_UP_FORT": 0.3838, "F1_DOWN_FORT": 0.37, "train_start": "2001-02-20", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VIX_Price_zscore_60d__div__hmm_p_stress", "vix_zscore_10d__div__heston_var_ev_h1", "NFCI_ret_5d__zrel__NEE_NextEra_ret_5d", "VIX_Price_zscore_60d__zrel__heston_var_ev_h3", "HangSeng_HK_ret_5d", "VIX_Price_zscore_60d__ret5x__COF_CapitalOne_ret_5d", "DUK_Duke_ret_5d__div__COF_CapitalOne_ret_5d", "DUK_Duke_ret_5d__ret5x__COF_CapitalOne_ret_5d", "EWL_Switzerland_vol_20d", "EWM_Malaysia_vol_20d", "EOG_EOGResources_vol_20d", "kalman_filtered__prod__VOD_Vodafone_vol_20d", "MSFT_ret_5d__div__NEE_NextEra_ret_5d", "GD_GeneralDynamics_zscore_60d"], "is_new": false}, {"model_id": "v2_h3_NORMAL_XGBoost_N15", "algo": "XGBoost", "regime": "NORMAL", "horizon": 3, "n_features": 15, "F1_dir": 0.5978, "F1_UP_FORT": 0.3706, "F1_DOWN_FORT": 0.3833, "train_start": "2001-02-20", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VIX_Price_zscore_60d__div__hmm_p_stress", "vix_zscore_10d__div__heston_var_ev_h1", "NFCI_ret_5d__zrel__NEE_NextEra_ret_5d", "VIX_Price_zscore_60d__zrel__heston_var_ev_h3", "HangSeng_HK_ret_5d", "VIX_Price_zscore_60d__ret5x__COF_CapitalOne_ret_5d", "DUK_Duke_ret_5d__div__COF_CapitalOne_ret_5d", "DUK_Duke_ret_5d__ret5x__COF_CapitalOne_ret_5d", "EWL_Switzerland_vol_20d", "EWM_Malaysia_vol_20d", "EOG_EOGResources_vol_20d", "kalman_filtered__prod__VOD_Vodafone_vol_20d", "MSFT_ret_5d__div__NEE_NextEra_ret_5d", "GD_GeneralDynamics_zscore_60d", "ENB_EnbridgeInc_ret_1d"], "is_new": false}, {"model_id": "v2_h3_NORMAL_XGBoost_N16", "algo": "XGBoost", "regime": "NORMAL", "horizon": 3, "n_features": 16, "F1_dir": 0.5974, "F1_UP_FORT": 0.3536, "F1_DOWN_FORT": 0.4126, "train_start": "2001-02-20", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VIX_Price_zscore_60d__div__hmm_p_stress", "vix_zscore_10d__div__heston_var_ev_h1", "NFCI_ret_5d__zrel__NEE_NextEra_ret_5d", "VIX_Price_zscore_60d__zrel__heston_var_ev_h3", "HangSeng_HK_ret_5d", "VIX_Price_zscore_60d__ret5x__COF_CapitalOne_ret_5d", "DUK_Duke_ret_5d__div__COF_CapitalOne_ret_5d", "DUK_Duke_ret_5d__ret5x__COF_CapitalOne_ret_5d", "EWL_Switzerland_vol_20d", "EWM_Malaysia_vol_20d", "EOG_EOGResources_vol_20d", "kalman_filtered__prod__VOD_Vodafone_vol_20d", "MSFT_ret_5d__div__NEE_NextEra_ret_5d", "GD_GeneralDynamics_zscore_60d", "ENB_EnbridgeInc_ret_1d", "INTC_ret_5d"], "is_new": false}, {"model_id": "v2_h3_NORMAL_XGBoost_N17", "algo": "XGBoost", "regime": "NORMAL", "horizon": 3, "n_features": 17, "F1_dir": 0.6053, "F1_UP_FORT": 0.3714, "F1_DOWN_FORT": 0.4099, "train_start": "2001-02-20", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VIX_Price_zscore_60d__div__hmm_p_stress", "vix_zscore_10d__div__heston_var_ev_h1", "NFCI_ret_5d__zrel__NEE_NextEra_ret_5d", "VIX_Price_zscore_60d__zrel__heston_var_ev_h3", "HangSeng_HK_ret_5d", "VIX_Price_zscore_60d__ret5x__COF_CapitalOne_ret_5d", "DUK_Duke_ret_5d__div__COF_CapitalOne_ret_5d", "DUK_Duke_ret_5d__ret5x__COF_CapitalOne_ret_5d", "EWL_Switzerland_vol_20d", "EWM_Malaysia_vol_20d", "EOG_EOGResources_vol_20d", "kalman_filtered__prod__VOD_Vodafone_vol_20d", "MSFT_ret_5d__div__NEE_NextEra_ret_5d", "GD_GeneralDynamics_zscore_60d", "ENB_EnbridgeInc_ret_1d", "INTC_ret_5d", "MSFT_ret_5d__zrel__AVB_AvalonBay_ret_5d"], "is_new": false}, {"model_id": "v2_h3_NORMAL_XGBoost_N18", "algo": "XGBoost", "regime": "NORMAL", "horizon": 3, "n_features": 18, "F1_dir": 0.5931, "F1_UP_FORT": 0.3958, "F1_DOWN_FORT": 0.3957, "train_start": "2001-02-20", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VIX_Price_zscore_60d__div__hmm_p_stress", "vix_zscore_10d__div__heston_var_ev_h1", "NFCI_ret_5d__zrel__NEE_NextEra_ret_5d", "VIX_Price_zscore_60d__zrel__heston_var_ev_h3", "HangSeng_HK_ret_5d", "VIX_Price_zscore_60d__ret5x__COF_CapitalOne_ret_5d", "DUK_Duke_ret_5d__div__COF_CapitalOne_ret_5d", "DUK_Duke_ret_5d__ret5x__COF_CapitalOne_ret_5d", "EWL_Switzerland_vol_20d", "EWM_Malaysia_vol_20d", "EOG_EOGResources_vol_20d", "kalman_filtered__prod__VOD_Vodafone_vol_20d", "MSFT_ret_5d__div__NEE_NextEra_ret_5d", "GD_GeneralDynamics_zscore_60d", "ENB_EnbridgeInc_ret_1d", "INTC_ret_5d", "MSFT_ret_5d__zrel__AVB_AvalonBay_ret_5d", "VIX_Price_zscore_60d__div__heston_var_ev_h3"], "is_new": false}, {"model_id": "v2_h3_NORMAL_XGBoost_N19", "algo": "XGBoost", "regime": "NORMAL", "horizon": 3, "n_features": 19, "F1_dir": 0.5767, "F1_UP_FORT": 0.3473, "F1_DOWN_FORT": 0.3915, "train_start": "2001-02-20", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VIX_Price_zscore_60d__div__hmm_p_stress", "vix_zscore_10d__div__heston_var_ev_h1", "NFCI_ret_5d__zrel__NEE_NextEra_ret_5d", "VIX_Price_zscore_60d__zrel__heston_var_ev_h3", "HangSeng_HK_ret_5d", "VIX_Price_zscore_60d__ret5x__COF_CapitalOne_ret_5d", "DUK_Duke_ret_5d__div__COF_CapitalOne_ret_5d", "DUK_Duke_ret_5d__ret5x__COF_CapitalOne_ret_5d", "EWL_Switzerland_vol_20d", "EWM_Malaysia_vol_20d", "EOG_EOGResources_vol_20d", "kalman_filtered__prod__VOD_Vodafone_vol_20d", "MSFT_ret_5d__div__NEE_NextEra_ret_5d", "GD_GeneralDynamics_zscore_60d", "ENB_EnbridgeInc_ret_1d", "INTC_ret_5d", "MSFT_ret_5d__zrel__AVB_AvalonBay_ret_5d", "VIX_Price_zscore_60d__div__heston_var_ev_h3", "US3M_Rate_zscore_60d"], "is_new": false}, {"model_id": "v2_h3_NORMAL_XGBoost_N20", "algo": "XGBoost", "regime": "NORMAL", "horizon": 3, "n_features": 20, "F1_dir": 0.593, "F1_UP_FORT": 0.3315, "F1_DOWN_FORT": 0.4094, "train_start": "2001-02-20", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VIX_Price_zscore_60d__div__hmm_p_stress", "vix_zscore_10d__div__heston_var_ev_h1", "NFCI_ret_5d__zrel__NEE_NextEra_ret_5d", "VIX_Price_zscore_60d__zrel__heston_var_ev_h3", "HangSeng_HK_ret_5d", "VIX_Price_zscore_60d__ret5x__COF_CapitalOne_ret_5d", "DUK_Duke_ret_5d__div__COF_CapitalOne_ret_5d", "DUK_Duke_ret_5d__ret5x__COF_CapitalOne_ret_5d", "EWL_Switzerland_vol_20d", "EWM_Malaysia_vol_20d", "EOG_EOGResources_vol_20d", "kalman_filtered__prod__VOD_Vodafone_vol_20d", "MSFT_ret_5d__div__NEE_NextEra_ret_5d", "GD_GeneralDynamics_zscore_60d", "ENB_EnbridgeInc_ret_1d", "INTC_ret_5d", "MSFT_ret_5d__zrel__AVB_AvalonBay_ret_5d", "VIX_Price_zscore_60d__div__heston_var_ev_h3", "US3M_Rate_zscore_60d", "kalman_filtered__minus__XLY_Disc_zscore_60d"], "is_new": false}, {"model_id": "v2_h3_NORMAL_XGBoost_N21", "algo": "XGBoost", "regime": "NORMAL", "horizon": 3, "n_features": 21, "F1_dir": 0.5862, "F1_UP_FORT": 0.3696, "F1_DOWN_FORT": 0.4321, "train_start": "2001-02-20", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VIX_Price_zscore_60d__div__hmm_p_stress", "vix_zscore_10d__div__heston_var_ev_h1", "NFCI_ret_5d__zrel__NEE_NextEra_ret_5d", "VIX_Price_zscore_60d__zrel__heston_var_ev_h3", "HangSeng_HK_ret_5d", "VIX_Price_zscore_60d__ret5x__COF_CapitalOne_ret_5d", "DUK_Duke_ret_5d__div__COF_CapitalOne_ret_5d", "DUK_Duke_ret_5d__ret5x__COF_CapitalOne_ret_5d", "EWL_Switzerland_vol_20d", "EWM_Malaysia_vol_20d", "EOG_EOGResources_vol_20d", "kalman_filtered__prod__VOD_Vodafone_vol_20d", "MSFT_ret_5d__div__NEE_NextEra_ret_5d", "GD_GeneralDynamics_zscore_60d", "ENB_EnbridgeInc_ret_1d", "INTC_ret_5d", "MSFT_ret_5d__zrel__AVB_AvalonBay_ret_5d", "VIX_Price_zscore_60d__div__heston_var_ev_h3", "US3M_Rate_zscore_60d", "kalman_filtered__minus__XLY_Disc_zscore_60d", "vix_zscore_10d__macross__MSFT_ret_5d"], "is_new": false}, {"model_id": "v2_h3_NORMAL_XGBoost_N22", "algo": "XGBoost", "regime": "NORMAL", "horizon": 3, "n_features": 22, "F1_dir": 0.5856, "F1_UP_FORT": 0.3694, "F1_DOWN_FORT": 0.3973, "train_start": "2001-02-20", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VIX_Price_zscore_60d__div__hmm_p_stress", "vix_zscore_10d__div__heston_var_ev_h1", "NFCI_ret_5d__zrel__NEE_NextEra_ret_5d", "VIX_Price_zscore_60d__zrel__heston_var_ev_h3", "HangSeng_HK_ret_5d", "VIX_Price_zscore_60d__ret5x__COF_CapitalOne_ret_5d", "DUK_Duke_ret_5d__div__COF_CapitalOne_ret_5d", "DUK_Duke_ret_5d__ret5x__COF_CapitalOne_ret_5d", "EWL_Switzerland_vol_20d", "EWM_Malaysia_vol_20d", "EOG_EOGResources_vol_20d", "kalman_filtered__prod__VOD_Vodafone_vol_20d", "MSFT_ret_5d__div__NEE_NextEra_ret_5d", "GD_GeneralDynamics_zscore_60d", "ENB_EnbridgeInc_ret_1d", "INTC_ret_5d", "MSFT_ret_5d__zrel__AVB_AvalonBay_ret_5d", "VIX_Price_zscore_60d__div__heston_var_ev_h3", "US3M_Rate_zscore_60d", "kalman_filtered__minus__XLY_Disc_zscore_60d", "vix_zscore_10d__macross__MSFT_ret_5d", "VOD_Vodafone_vol_20d__macross__COF_CapitalOne_ret_5d"], "is_new": false}, {"model_id": "v2_h3_NORMAL_LightGBM_N5", "algo": "LightGBM", "regime": "NORMAL", "horizon": 3, "n_features": 5, "F1_dir": 0.5347, "F1_UP_FORT": 0.3206, "F1_DOWN_FORT": 0.3516, "train_start": "2001-02-20", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VIX_Price_zscore_60d__div__hmm_p_stress", "vix_zscore_10d__div__heston_var_ev_h1", "NFCI_ret_5d__zrel__NEE_NextEra_ret_5d", "VIX_Price_zscore_60d__zrel__heston_var_ev_h3", "HangSeng_HK_ret_5d"], "is_new": false}, {"model_id": "v2_h3_NORMAL_LightGBM_N6", "algo": "LightGBM", "regime": "NORMAL", "horizon": 3, "n_features": 6, "F1_dir": 0.5244, "F1_UP_FORT": 0.3498, "F1_DOWN_FORT": 0.3341, "train_start": "2001-02-20", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VIX_Price_zscore_60d__div__hmm_p_stress", "vix_zscore_10d__div__heston_var_ev_h1", "NFCI_ret_5d__zrel__NEE_NextEra_ret_5d", "VIX_Price_zscore_60d__zrel__heston_var_ev_h3", "HangSeng_HK_ret_5d", "VIX_Price_zscore_60d__ret5x__COF_CapitalOne_ret_5d"], "is_new": false}, {"model_id": "v2_h3_NORMAL_LightGBM_N7", "algo": "LightGBM", "regime": "NORMAL", "horizon": 3, "n_features": 7, "F1_dir": 0.5253, "F1_UP_FORT": 0.3264, "F1_DOWN_FORT": 0.3497, "train_start": "2001-02-20", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VIX_Price_zscore_60d__div__hmm_p_stress", "vix_zscore_10d__div__heston_var_ev_h1", "NFCI_ret_5d__zrel__NEE_NextEra_ret_5d", "VIX_Price_zscore_60d__zrel__heston_var_ev_h3", "HangSeng_HK_ret_5d", "VIX_Price_zscore_60d__ret5x__COF_CapitalOne_ret_5d", "DUK_Duke_ret_5d__div__COF_CapitalOne_ret_5d"], "is_new": false}, {"model_id": "v2_h3_NORMAL_LightGBM_N8", "algo": "LightGBM", "regime": "NORMAL", "horizon": 3, "n_features": 8, "F1_dir": 0.5349, "F1_UP_FORT": 0.3242, "F1_DOWN_FORT": 0.3442, "train_start": "2001-02-20", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VIX_Price_zscore_60d__div__hmm_p_stress", "vix_zscore_10d__div__heston_var_ev_h1", "NFCI_ret_5d__zrel__NEE_NextEra_ret_5d", "VIX_Price_zscore_60d__zrel__heston_var_ev_h3", "HangSeng_HK_ret_5d", "VIX_Price_zscore_60d__ret5x__COF_CapitalOne_ret_5d", "DUK_Duke_ret_5d__div__COF_CapitalOne_ret_5d", "DUK_Duke_ret_5d__ret5x__COF_CapitalOne_ret_5d"], "is_new": false}, {"model_id": "v2_h3_NORMAL_LightGBM_N9", "algo": "LightGBM", "regime": "NORMAL", "horizon": 3, "n_features": 9, "F1_dir": 0.5497, "F1_UP_FORT": 0.3526, "F1_DOWN_FORT": 0.3519, "train_start": "2001-02-20", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VIX_Price_zscore_60d__div__hmm_p_stress", "vix_zscore_10d__div__heston_var_ev_h1", "NFCI_ret_5d__zrel__NEE_NextEra_ret_5d", "VIX_Price_zscore_60d__zrel__heston_var_ev_h3", "HangSeng_HK_ret_5d", "VIX_Price_zscore_60d__ret5x__COF_CapitalOne_ret_5d", "DUK_Duke_ret_5d__div__COF_CapitalOne_ret_5d", "DUK_Duke_ret_5d__ret5x__COF_CapitalOne_ret_5d", "EWL_Switzerland_vol_20d"], "is_new": false}, {"model_id": "v2_h3_NORMAL_LightGBM_N10", "algo": "LightGBM", "regime": "NORMAL", "horizon": 3, "n_features": 10, "F1_dir": 0.5367, "F1_UP_FORT": 0.3558, "F1_DOWN_FORT": 0.3441, "train_start": "2001-02-20", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VIX_Price_zscore_60d__div__hmm_p_stress", "vix_zscore_10d__div__heston_var_ev_h1", "NFCI_ret_5d__zrel__NEE_NextEra_ret_5d", "VIX_Price_zscore_60d__zrel__heston_var_ev_h3", "HangSeng_HK_ret_5d", "VIX_Price_zscore_60d__ret5x__COF_CapitalOne_ret_5d", "DUK_Duke_ret_5d__div__COF_CapitalOne_ret_5d", "DUK_Duke_ret_5d__ret5x__COF_CapitalOne_ret_5d", "EWL_Switzerland_vol_20d", "EWM_Malaysia_vol_20d"], "is_new": false}, {"model_id": "v2_h3_NORMAL_LightGBM_N11", "algo": "LightGBM", "regime": "NORMAL", "horizon": 3, "n_features": 11, "F1_dir": 0.5582, "F1_UP_FORT": 0.3469, "F1_DOWN_FORT": 0.3624, "train_start": "2001-02-20", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VIX_Price_zscore_60d__div__hmm_p_stress", "vix_zscore_10d__div__heston_var_ev_h1", "NFCI_ret_5d__zrel__NEE_NextEra_ret_5d", "VIX_Price_zscore_60d__zrel__heston_var_ev_h3", "HangSeng_HK_ret_5d", "VIX_Price_zscore_60d__ret5x__COF_CapitalOne_ret_5d", "DUK_Duke_ret_5d__div__COF_CapitalOne_ret_5d", "DUK_Duke_ret_5d__ret5x__COF_CapitalOne_ret_5d", "EWL_Switzerland_vol_20d", "EWM_Malaysia_vol_20d", "EOG_EOGResources_vol_20d"], "is_new": false}, {"model_id": "v2_h3_NORMAL_LightGBM_N12", "algo": "LightGBM", "regime": "NORMAL", "horizon": 3, "n_features": 12, "F1_dir": 0.5649, "F1_UP_FORT": 0.2995, "F1_DOWN_FORT": 0.3415, "train_start": "2001-02-20", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VIX_Price_zscore_60d__div__hmm_p_stress", "vix_zscore_10d__div__heston_var_ev_h1", "NFCI_ret_5d__zrel__NEE_NextEra_ret_5d", "VIX_Price_zscore_60d__zrel__heston_var_ev_h3", "HangSeng_HK_ret_5d", "VIX_Price_zscore_60d__ret5x__COF_CapitalOne_ret_5d", "DUK_Duke_ret_5d__div__COF_CapitalOne_ret_5d", "DUK_Duke_ret_5d__ret5x__COF_CapitalOne_ret_5d", "EWL_Switzerland_vol_20d", "EWM_Malaysia_vol_20d", "EOG_EOGResources_vol_20d", "kalman_filtered__prod__VOD_Vodafone_vol_20d"], "is_new": false}, {"model_id": "v2_h3_NORMAL_LightGBM_N13", "algo": "LightGBM", "regime": "NORMAL", "horizon": 3, "n_features": 13, "F1_dir": 0.5739, "F1_UP_FORT": 0.3562, "F1_DOWN_FORT": 0.3493, "train_start": "2001-02-20", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VIX_Price_zscore_60d__div__hmm_p_stress", "vix_zscore_10d__div__heston_var_ev_h1", "NFCI_ret_5d__zrel__NEE_NextEra_ret_5d", "VIX_Price_zscore_60d__zrel__heston_var_ev_h3", "HangSeng_HK_ret_5d", "VIX_Price_zscore_60d__ret5x__COF_CapitalOne_ret_5d", "DUK_Duke_ret_5d__div__COF_CapitalOne_ret_5d", "DUK_Duke_ret_5d__ret5x__COF_CapitalOne_ret_5d", "EWL_Switzerland_vol_20d", "EWM_Malaysia_vol_20d", "EOG_EOGResources_vol_20d", "kalman_filtered__prod__VOD_Vodafone_vol_20d", "MSFT_ret_5d__div__NEE_NextEra_ret_5d"], "is_new": false}, {"model_id": "v2_h3_NORMAL_LightGBM_N14", "algo": "LightGBM", "regime": "NORMAL", "horizon": 3, "n_features": 14, "F1_dir": 0.5635, "F1_UP_FORT": 0.3402, "F1_DOWN_FORT": 0.3801, "train_start": "2001-02-20", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VIX_Price_zscore_60d__div__hmm_p_stress", "vix_zscore_10d__div__heston_var_ev_h1", "NFCI_ret_5d__zrel__NEE_NextEra_ret_5d", "VIX_Price_zscore_60d__zrel__heston_var_ev_h3", "HangSeng_HK_ret_5d", "VIX_Price_zscore_60d__ret5x__COF_CapitalOne_ret_5d", "DUK_Duke_ret_5d__div__COF_CapitalOne_ret_5d", "DUK_Duke_ret_5d__ret5x__COF_CapitalOne_ret_5d", "EWL_Switzerland_vol_20d", "EWM_Malaysia_vol_20d", "EOG_EOGResources_vol_20d", "kalman_filtered__prod__VOD_Vodafone_vol_20d", "MSFT_ret_5d__div__NEE_NextEra_ret_5d", "GD_GeneralDynamics_zscore_60d"], "is_new": false}, {"model_id": "v2_h3_NORMAL_LightGBM_N15", "algo": "LightGBM", "regime": "NORMAL", "horizon": 3, "n_features": 15, "F1_dir": 0.5928, "F1_UP_FORT": 0.3679, "F1_DOWN_FORT": 0.3829, "train_start": "2001-02-20", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VIX_Price_zscore_60d__div__hmm_p_stress", "vix_zscore_10d__div__heston_var_ev_h1", "NFCI_ret_5d__zrel__NEE_NextEra_ret_5d", "VIX_Price_zscore_60d__zrel__heston_var_ev_h3", "HangSeng_HK_ret_5d", "VIX_Price_zscore_60d__ret5x__COF_CapitalOne_ret_5d", "DUK_Duke_ret_5d__div__COF_CapitalOne_ret_5d", "DUK_Duke_ret_5d__ret5x__COF_CapitalOne_ret_5d", "EWL_Switzerland_vol_20d", "EWM_Malaysia_vol_20d", "EOG_EOGResources_vol_20d", "kalman_filtered__prod__VOD_Vodafone_vol_20d", "MSFT_ret_5d__div__NEE_NextEra_ret_5d", "GD_GeneralDynamics_zscore_60d", "ENB_EnbridgeInc_ret_1d"], "is_new": false}, {"model_id": "v2_h3_NORMAL_LightGBM_N16", "algo": "LightGBM", "regime": "NORMAL", "horizon": 3, "n_features": 16, "F1_dir": 0.5798, "F1_UP_FORT": 0.3378, "F1_DOWN_FORT": 0.3934, "train_start": "2001-02-20", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VIX_Price_zscore_60d__div__hmm_p_stress", "vix_zscore_10d__div__heston_var_ev_h1", "NFCI_ret_5d__zrel__NEE_NextEra_ret_5d", "VIX_Price_zscore_60d__zrel__heston_var_ev_h3", "HangSeng_HK_ret_5d", "VIX_Price_zscore_60d__ret5x__COF_CapitalOne_ret_5d", "DUK_Duke_ret_5d__div__COF_CapitalOne_ret_5d", "DUK_Duke_ret_5d__ret5x__COF_CapitalOne_ret_5d", "EWL_Switzerland_vol_20d", "EWM_Malaysia_vol_20d", "EOG_EOGResources_vol_20d", "kalman_filtered__prod__VOD_Vodafone_vol_20d", "MSFT_ret_5d__div__NEE_NextEra_ret_5d", "GD_GeneralDynamics_zscore_60d", "ENB_EnbridgeInc_ret_1d", "INTC_ret_5d"], "is_new": false}, {"model_id": "v2_h3_NORMAL_LightGBM_N17", "algo": "LightGBM", "regime": "NORMAL", "horizon": 3, "n_features": 17, "F1_dir": 0.6, "F1_UP_FORT": 0.3616, "F1_DOWN_FORT": 0.3881, "train_start": "2001-02-20", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VIX_Price_zscore_60d__div__hmm_p_stress", "vix_zscore_10d__div__heston_var_ev_h1", "NFCI_ret_5d__zrel__NEE_NextEra_ret_5d", "VIX_Price_zscore_60d__zrel__heston_var_ev_h3", "HangSeng_HK_ret_5d", "VIX_Price_zscore_60d__ret5x__COF_CapitalOne_ret_5d", "DUK_Duke_ret_5d__div__COF_CapitalOne_ret_5d", "DUK_Duke_ret_5d__ret5x__COF_CapitalOne_ret_5d", "EWL_Switzerland_vol_20d", "EWM_Malaysia_vol_20d", "EOG_EOGResources_vol_20d", "kalman_filtered__prod__VOD_Vodafone_vol_20d", "MSFT_ret_5d__div__NEE_NextEra_ret_5d", "GD_GeneralDynamics_zscore_60d", "ENB_EnbridgeInc_ret_1d", "INTC_ret_5d", "MSFT_ret_5d__zrel__AVB_AvalonBay_ret_5d"], "is_new": false}, {"model_id": "v2_h3_NORMAL_LightGBM_N18", "algo": "LightGBM", "regime": "NORMAL", "horizon": 3, "n_features": 18, "F1_dir": 0.5846, "F1_UP_FORT": 0.3464, "F1_DOWN_FORT": 0.3732, "train_start": "2001-02-20", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VIX_Price_zscore_60d__div__hmm_p_stress", "vix_zscore_10d__div__heston_var_ev_h1", "NFCI_ret_5d__zrel__NEE_NextEra_ret_5d", "VIX_Price_zscore_60d__zrel__heston_var_ev_h3", "HangSeng_HK_ret_5d", "VIX_Price_zscore_60d__ret5x__COF_CapitalOne_ret_5d", "DUK_Duke_ret_5d__div__COF_CapitalOne_ret_5d", "DUK_Duke_ret_5d__ret5x__COF_CapitalOne_ret_5d", "EWL_Switzerland_vol_20d", "EWM_Malaysia_vol_20d", "EOG_EOGResources_vol_20d", "kalman_filtered__prod__VOD_Vodafone_vol_20d", "MSFT_ret_5d__div__NEE_NextEra_ret_5d", "GD_GeneralDynamics_zscore_60d", "ENB_EnbridgeInc_ret_1d", "INTC_ret_5d", "MSFT_ret_5d__zrel__AVB_AvalonBay_ret_5d", "VIX_Price_zscore_60d__div__heston_var_ev_h3"], "is_new": false}, {"model_id": "v2_h3_NORMAL_LightGBM_N19", "algo": "LightGBM", "regime": "NORMAL", "horizon": 3, "n_features": 19, "F1_dir": 0.5653, "F1_UP_FORT": 0.3371, "F1_DOWN_FORT": 0.4093, "train_start": "2001-02-20", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VIX_Price_zscore_60d__div__hmm_p_stress", "vix_zscore_10d__div__heston_var_ev_h1", "NFCI_ret_5d__zrel__NEE_NextEra_ret_5d", "VIX_Price_zscore_60d__zrel__heston_var_ev_h3", "HangSeng_HK_ret_5d", "VIX_Price_zscore_60d__ret5x__COF_CapitalOne_ret_5d", "DUK_Duke_ret_5d__div__COF_CapitalOne_ret_5d", "DUK_Duke_ret_5d__ret5x__COF_CapitalOne_ret_5d", "EWL_Switzerland_vol_20d", "EWM_Malaysia_vol_20d", "EOG_EOGResources_vol_20d", "kalman_filtered__prod__VOD_Vodafone_vol_20d", "MSFT_ret_5d__div__NEE_NextEra_ret_5d", "GD_GeneralDynamics_zscore_60d", "ENB_EnbridgeInc_ret_1d", "INTC_ret_5d", "MSFT_ret_5d__zrel__AVB_AvalonBay_ret_5d", "VIX_Price_zscore_60d__div__heston_var_ev_h3", "US3M_Rate_zscore_60d"], "is_new": false}, {"model_id": "v2_h3_NORMAL_LightGBM_N20", "algo": "LightGBM", "regime": "NORMAL", "horizon": 3, "n_features": 20, "F1_dir": 0.5865, "F1_UP_FORT": 0.3257, "F1_DOWN_FORT": 0.4148, "train_start": "2001-02-20", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VIX_Price_zscore_60d__div__hmm_p_stress", "vix_zscore_10d__div__heston_var_ev_h1", "NFCI_ret_5d__zrel__NEE_NextEra_ret_5d", "VIX_Price_zscore_60d__zrel__heston_var_ev_h3", "HangSeng_HK_ret_5d", "VIX_Price_zscore_60d__ret5x__COF_CapitalOne_ret_5d", "DUK_Duke_ret_5d__div__COF_CapitalOne_ret_5d", "DUK_Duke_ret_5d__ret5x__COF_CapitalOne_ret_5d", "EWL_Switzerland_vol_20d", "EWM_Malaysia_vol_20d", "EOG_EOGResources_vol_20d", "kalman_filtered__prod__VOD_Vodafone_vol_20d", "MSFT_ret_5d__div__NEE_NextEra_ret_5d", "GD_GeneralDynamics_zscore_60d", "ENB_EnbridgeInc_ret_1d", "INTC_ret_5d", "MSFT_ret_5d__zrel__AVB_AvalonBay_ret_5d", "VIX_Price_zscore_60d__div__heston_var_ev_h3", "US3M_Rate_zscore_60d", "kalman_filtered__minus__XLY_Disc_zscore_60d"], "is_new": false}, {"model_id": "v2_h3_NORMAL_LightGBM_N21", "algo": "LightGBM", "regime": "NORMAL", "horizon": 3, "n_features": 21, "F1_dir": 0.5878, "F1_UP_FORT": 0.2989, "F1_DOWN_FORT": 0.4339, "train_start": "2001-02-20", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VIX_Price_zscore_60d__div__hmm_p_stress", "vix_zscore_10d__div__heston_var_ev_h1", "NFCI_ret_5d__zrel__NEE_NextEra_ret_5d", "VIX_Price_zscore_60d__zrel__heston_var_ev_h3", "HangSeng_HK_ret_5d", "VIX_Price_zscore_60d__ret5x__COF_CapitalOne_ret_5d", "DUK_Duke_ret_5d__div__COF_CapitalOne_ret_5d", "DUK_Duke_ret_5d__ret5x__COF_CapitalOne_ret_5d", "EWL_Switzerland_vol_20d", "EWM_Malaysia_vol_20d", "EOG_EOGResources_vol_20d", "kalman_filtered__prod__VOD_Vodafone_vol_20d", "MSFT_ret_5d__div__NEE_NextEra_ret_5d", "GD_GeneralDynamics_zscore_60d", "ENB_EnbridgeInc_ret_1d", "INTC_ret_5d", "MSFT_ret_5d__zrel__AVB_AvalonBay_ret_5d", "VIX_Price_zscore_60d__div__heston_var_ev_h3", "US3M_Rate_zscore_60d", "kalman_filtered__minus__XLY_Disc_zscore_60d", "vix_zscore_10d__macross__MSFT_ret_5d"], "is_new": false}, {"model_id": "v2_h3_NORMAL_LightGBM_N22", "algo": "LightGBM", "regime": "NORMAL", "horizon": 3, "n_features": 22, "F1_dir": 0.5792, "F1_UP_FORT": 0.3343, "F1_DOWN_FORT": 0.4115, "train_start": "2001-02-20", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VIX_Price_zscore_60d__div__hmm_p_stress", "vix_zscore_10d__div__heston_var_ev_h1", "NFCI_ret_5d__zrel__NEE_NextEra_ret_5d", "VIX_Price_zscore_60d__zrel__heston_var_ev_h3", "HangSeng_HK_ret_5d", "VIX_Price_zscore_60d__ret5x__COF_CapitalOne_ret_5d", "DUK_Duke_ret_5d__div__COF_CapitalOne_ret_5d", "DUK_Duke_ret_5d__ret5x__COF_CapitalOne_ret_5d", "EWL_Switzerland_vol_20d", "EWM_Malaysia_vol_20d", "EOG_EOGResources_vol_20d", "kalman_filtered__prod__VOD_Vodafone_vol_20d", "MSFT_ret_5d__div__NEE_NextEra_ret_5d", "GD_GeneralDynamics_zscore_60d", "ENB_EnbridgeInc_ret_1d", "INTC_ret_5d", "MSFT_ret_5d__zrel__AVB_AvalonBay_ret_5d", "VIX_Price_zscore_60d__div__heston_var_ev_h3", "US3M_Rate_zscore_60d", "kalman_filtered__minus__XLY_Disc_zscore_60d", "vix_zscore_10d__macross__MSFT_ret_5d", "VOD_Vodafone_vol_20d__macross__COF_CapitalOne_ret_5d"], "is_new": false}, {"model_id": "v2_h3_NORMAL_GradientBoosting_N5", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 3, "n_features": 5, "F1_dir": 0.5505, "F1_UP_FORT": 0.3364, "F1_DOWN_FORT": 0.3548, "train_start": "2001-02-20", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VIX_Price_zscore_60d__div__hmm_p_stress", "vix_zscore_10d__div__heston_var_ev_h1", "NFCI_ret_5d__zrel__NEE_NextEra_ret_5d", "VIX_Price_zscore_60d__zrel__heston_var_ev_h3", "HangSeng_HK_ret_5d"], "is_new": false}, {"model_id": "v2_h3_NORMAL_GradientBoosting_N6", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 3, "n_features": 6, "F1_dir": 0.5645, "F1_UP_FORT": 0.3823, "F1_DOWN_FORT": 0.3632, "train_start": "2001-02-20", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VIX_Price_zscore_60d__div__hmm_p_stress", "vix_zscore_10d__div__heston_var_ev_h1", "NFCI_ret_5d__zrel__NEE_NextEra_ret_5d", "VIX_Price_zscore_60d__zrel__heston_var_ev_h3", "HangSeng_HK_ret_5d", "VIX_Price_zscore_60d__ret5x__COF_CapitalOne_ret_5d"], "is_new": false}, {"model_id": "v2_h3_NORMAL_GradientBoosting_N7", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 3, "n_features": 7, "F1_dir": 0.549, "F1_UP_FORT": 0.3276, "F1_DOWN_FORT": 0.3401, "train_start": "2001-02-20", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VIX_Price_zscore_60d__div__hmm_p_stress", "vix_zscore_10d__div__heston_var_ev_h1", "NFCI_ret_5d__zrel__NEE_NextEra_ret_5d", "VIX_Price_zscore_60d__zrel__heston_var_ev_h3", "HangSeng_HK_ret_5d", "VIX_Price_zscore_60d__ret5x__COF_CapitalOne_ret_5d", "DUK_Duke_ret_5d__div__COF_CapitalOne_ret_5d"], "is_new": false}, {"model_id": "v2_h3_NORMAL_GradientBoosting_N8", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 3, "n_features": 8, "F1_dir": 0.5505, "F1_UP_FORT": 0.3512, "F1_DOWN_FORT": 0.3597, "train_start": "2001-02-20", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VIX_Price_zscore_60d__div__hmm_p_stress", "vix_zscore_10d__div__heston_var_ev_h1", "NFCI_ret_5d__zrel__NEE_NextEra_ret_5d", "VIX_Price_zscore_60d__zrel__heston_var_ev_h3", "HangSeng_HK_ret_5d", "VIX_Price_zscore_60d__ret5x__COF_CapitalOne_ret_5d", "DUK_Duke_ret_5d__div__COF_CapitalOne_ret_5d", "DUK_Duke_ret_5d__ret5x__COF_CapitalOne_ret_5d"], "is_new": false}, {"model_id": "v2_h3_NORMAL_GradientBoosting_N9", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 3, "n_features": 9, "F1_dir": 0.541, "F1_UP_FORT": 0.3105, "F1_DOWN_FORT": 0.3029, "train_start": "2001-02-20", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VIX_Price_zscore_60d__div__hmm_p_stress", "vix_zscore_10d__div__heston_var_ev_h1", "NFCI_ret_5d__zrel__NEE_NextEra_ret_5d", "VIX_Price_zscore_60d__zrel__heston_var_ev_h3", "HangSeng_HK_ret_5d", "VIX_Price_zscore_60d__ret5x__COF_CapitalOne_ret_5d", "DUK_Duke_ret_5d__div__COF_CapitalOne_ret_5d", "DUK_Duke_ret_5d__ret5x__COF_CapitalOne_ret_5d", "EWL_Switzerland_vol_20d"], "is_new": false}, {"model_id": "v2_h3_NORMAL_GradientBoosting_N10", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 3, "n_features": 10, "F1_dir": 0.5532, "F1_UP_FORT": 0.33, "F1_DOWN_FORT": 0.3576, "train_start": "2001-02-20", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VIX_Price_zscore_60d__div__hmm_p_stress", "vix_zscore_10d__div__heston_var_ev_h1", "NFCI_ret_5d__zrel__NEE_NextEra_ret_5d", "VIX_Price_zscore_60d__zrel__heston_var_ev_h3", "HangSeng_HK_ret_5d", "VIX_Price_zscore_60d__ret5x__COF_CapitalOne_ret_5d", "DUK_Duke_ret_5d__div__COF_CapitalOne_ret_5d", "DUK_Duke_ret_5d__ret5x__COF_CapitalOne_ret_5d", "EWL_Switzerland_vol_20d", "EWM_Malaysia_vol_20d"], "is_new": false}, {"model_id": "v2_h3_NORMAL_GradientBoosting_N11", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 3, "n_features": 11, "F1_dir": 0.5722, "F1_UP_FORT": 0.355, "F1_DOWN_FORT": 0.3382, "train_start": "2001-02-20", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VIX_Price_zscore_60d__div__hmm_p_stress", "vix_zscore_10d__div__heston_var_ev_h1", "NFCI_ret_5d__zrel__NEE_NextEra_ret_5d", "VIX_Price_zscore_60d__zrel__heston_var_ev_h3", "HangSeng_HK_ret_5d", "VIX_Price_zscore_60d__ret5x__COF_CapitalOne_ret_5d", "DUK_Duke_ret_5d__div__COF_CapitalOne_ret_5d", "DUK_Duke_ret_5d__ret5x__COF_CapitalOne_ret_5d", "EWL_Switzerland_vol_20d", "EWM_Malaysia_vol_20d", "EOG_EOGResources_vol_20d"], "is_new": false}, {"model_id": "v2_h3_NORMAL_GradientBoosting_N12", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 3, "n_features": 12, "F1_dir": 0.5726, "F1_UP_FORT": 0.3594, "F1_DOWN_FORT": 0.3547, "train_start": "2001-02-20", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VIX_Price_zscore_60d__div__hmm_p_stress", "vix_zscore_10d__div__heston_var_ev_h1", "NFCI_ret_5d__zrel__NEE_NextEra_ret_5d", "VIX_Price_zscore_60d__zrel__heston_var_ev_h3", "HangSeng_HK_ret_5d", "VIX_Price_zscore_60d__ret5x__COF_CapitalOne_ret_5d", "DUK_Duke_ret_5d__div__COF_CapitalOne_ret_5d", "DUK_Duke_ret_5d__ret5x__COF_CapitalOne_ret_5d", "EWL_Switzerland_vol_20d", "EWM_Malaysia_vol_20d", "EOG_EOGResources_vol_20d", "kalman_filtered__prod__VOD_Vodafone_vol_20d"], "is_new": false}, {"model_id": "v2_h3_NORMAL_GradientBoosting_N13", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 3, "n_features": 13, "F1_dir": 0.5638, "F1_UP_FORT": 0.3487, "F1_DOWN_FORT": 0.362, "train_start": "2001-02-20", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VIX_Price_zscore_60d__div__hmm_p_stress", "vix_zscore_10d__div__heston_var_ev_h1", "NFCI_ret_5d__zrel__NEE_NextEra_ret_5d", "VIX_Price_zscore_60d__zrel__heston_var_ev_h3", "HangSeng_HK_ret_5d", "VIX_Price_zscore_60d__ret5x__COF_CapitalOne_ret_5d", "DUK_Duke_ret_5d__div__COF_CapitalOne_ret_5d", "DUK_Duke_ret_5d__ret5x__COF_CapitalOne_ret_5d", "EWL_Switzerland_vol_20d", "EWM_Malaysia_vol_20d", "EOG_EOGResources_vol_20d", "kalman_filtered__prod__VOD_Vodafone_vol_20d", "MSFT_ret_5d__div__NEE_NextEra_ret_5d"], "is_new": false}, {"model_id": "v2_h3_NORMAL_GradientBoosting_N14", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 3, "n_features": 14, "F1_dir": 0.582, "F1_UP_FORT": 0.3724, "F1_DOWN_FORT": 0.3973, "train_start": "2001-02-20", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VIX_Price_zscore_60d__div__hmm_p_stress", "vix_zscore_10d__div__heston_var_ev_h1", "NFCI_ret_5d__zrel__NEE_NextEra_ret_5d", "VIX_Price_zscore_60d__zrel__heston_var_ev_h3", "HangSeng_HK_ret_5d", "VIX_Price_zscore_60d__ret5x__COF_CapitalOne_ret_5d", "DUK_Duke_ret_5d__div__COF_CapitalOne_ret_5d", "DUK_Duke_ret_5d__ret5x__COF_CapitalOne_ret_5d", "EWL_Switzerland_vol_20d", "EWM_Malaysia_vol_20d", "EOG_EOGResources_vol_20d", "kalman_filtered__prod__VOD_Vodafone_vol_20d", "MSFT_ret_5d__div__NEE_NextEra_ret_5d", "GD_GeneralDynamics_zscore_60d"], "is_new": false}, {"model_id": "v2_h3_NORMAL_GradientBoosting_N15", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 3, "n_features": 15, "F1_dir": 0.5931, "F1_UP_FORT": 0.3673, "F1_DOWN_FORT": 0.3982, "train_start": "2001-02-20", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VIX_Price_zscore_60d__div__hmm_p_stress", "vix_zscore_10d__div__heston_var_ev_h1", "NFCI_ret_5d__zrel__NEE_NextEra_ret_5d", "VIX_Price_zscore_60d__zrel__heston_var_ev_h3", "HangSeng_HK_ret_5d", "VIX_Price_zscore_60d__ret5x__COF_CapitalOne_ret_5d", "DUK_Duke_ret_5d__div__COF_CapitalOne_ret_5d", "DUK_Duke_ret_5d__ret5x__COF_CapitalOne_ret_5d", "EWL_Switzerland_vol_20d", "EWM_Malaysia_vol_20d", "EOG_EOGResources_vol_20d", "kalman_filtered__prod__VOD_Vodafone_vol_20d", "MSFT_ret_5d__div__NEE_NextEra_ret_5d", "GD_GeneralDynamics_zscore_60d", "ENB_EnbridgeInc_ret_1d"], "is_new": false}, {"model_id": "v2_h3_NORMAL_GradientBoosting_N16", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 3, "n_features": 16, "F1_dir": 0.6023, "F1_UP_FORT": 0.3717, "F1_DOWN_FORT": 0.3864, "train_start": "2001-02-20", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VIX_Price_zscore_60d__div__hmm_p_stress", "vix_zscore_10d__div__heston_var_ev_h1", "NFCI_ret_5d__zrel__NEE_NextEra_ret_5d", "VIX_Price_zscore_60d__zrel__heston_var_ev_h3", "HangSeng_HK_ret_5d", "VIX_Price_zscore_60d__ret5x__COF_CapitalOne_ret_5d", "DUK_Duke_ret_5d__div__COF_CapitalOne_ret_5d", "DUK_Duke_ret_5d__ret5x__COF_CapitalOne_ret_5d", "EWL_Switzerland_vol_20d", "EWM_Malaysia_vol_20d", "EOG_EOGResources_vol_20d", "kalman_filtered__prod__VOD_Vodafone_vol_20d", "MSFT_ret_5d__div__NEE_NextEra_ret_5d", "GD_GeneralDynamics_zscore_60d", "ENB_EnbridgeInc_ret_1d", "INTC_ret_5d"], "is_new": false}, {"model_id": "v2_h3_NORMAL_GradientBoosting_N17", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 3, "n_features": 17, "F1_dir": 0.6123, "F1_UP_FORT": 0.3686, "F1_DOWN_FORT": 0.4161, "train_start": "2001-02-20", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VIX_Price_zscore_60d__div__hmm_p_stress", "vix_zscore_10d__div__heston_var_ev_h1", "NFCI_ret_5d__zrel__NEE_NextEra_ret_5d", "VIX_Price_zscore_60d__zrel__heston_var_ev_h3", "HangSeng_HK_ret_5d", "VIX_Price_zscore_60d__ret5x__COF_CapitalOne_ret_5d", "DUK_Duke_ret_5d__div__COF_CapitalOne_ret_5d", "DUK_Duke_ret_5d__ret5x__COF_CapitalOne_ret_5d", "EWL_Switzerland_vol_20d", "EWM_Malaysia_vol_20d", "EOG_EOGResources_vol_20d", "kalman_filtered__prod__VOD_Vodafone_vol_20d", "MSFT_ret_5d__div__NEE_NextEra_ret_5d", "GD_GeneralDynamics_zscore_60d", "ENB_EnbridgeInc_ret_1d", "INTC_ret_5d", "MSFT_ret_5d__zrel__AVB_AvalonBay_ret_5d"], "is_new": false}, {"model_id": "v2_h3_NORMAL_GradientBoosting_N18", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 3, "n_features": 18, "F1_dir": 0.5894, "F1_UP_FORT": 0.3516, "F1_DOWN_FORT": 0.3949, "train_start": "2001-02-20", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VIX_Price_zscore_60d__div__hmm_p_stress", "vix_zscore_10d__div__heston_var_ev_h1", "NFCI_ret_5d__zrel__NEE_NextEra_ret_5d", "VIX_Price_zscore_60d__zrel__heston_var_ev_h3", "HangSeng_HK_ret_5d", "VIX_Price_zscore_60d__ret5x__COF_CapitalOne_ret_5d", "DUK_Duke_ret_5d__div__COF_CapitalOne_ret_5d", "DUK_Duke_ret_5d__ret5x__COF_CapitalOne_ret_5d", "EWL_Switzerland_vol_20d", "EWM_Malaysia_vol_20d", "EOG_EOGResources_vol_20d", "kalman_filtered__prod__VOD_Vodafone_vol_20d", "MSFT_ret_5d__div__NEE_NextEra_ret_5d", "GD_GeneralDynamics_zscore_60d", "ENB_EnbridgeInc_ret_1d", "INTC_ret_5d", "MSFT_ret_5d__zrel__AVB_AvalonBay_ret_5d", "VIX_Price_zscore_60d__div__heston_var_ev_h3"], "is_new": false}, {"model_id": "v2_h3_NORMAL_GradientBoosting_N19", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 3, "n_features": 19, "F1_dir": 0.5828, "F1_UP_FORT": 0.3579, "F1_DOWN_FORT": 0.4063, "train_start": "2001-02-20", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VIX_Price_zscore_60d__div__hmm_p_stress", "vix_zscore_10d__div__heston_var_ev_h1", "NFCI_ret_5d__zrel__NEE_NextEra_ret_5d", "VIX_Price_zscore_60d__zrel__heston_var_ev_h3", "HangSeng_HK_ret_5d", "VIX_Price_zscore_60d__ret5x__COF_CapitalOne_ret_5d", "DUK_Duke_ret_5d__div__COF_CapitalOne_ret_5d", "DUK_Duke_ret_5d__ret5x__COF_CapitalOne_ret_5d", "EWL_Switzerland_vol_20d", "EWM_Malaysia_vol_20d", "EOG_EOGResources_vol_20d", "kalman_filtered__prod__VOD_Vodafone_vol_20d", "MSFT_ret_5d__div__NEE_NextEra_ret_5d", "GD_GeneralDynamics_zscore_60d", "ENB_EnbridgeInc_ret_1d", "INTC_ret_5d", "MSFT_ret_5d__zrel__AVB_AvalonBay_ret_5d", "VIX_Price_zscore_60d__div__heston_var_ev_h3", "US3M_Rate_zscore_60d"], "is_new": false}, {"model_id": "v2_h3_NORMAL_GradientBoosting_N20", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 3, "n_features": 20, "F1_dir": 0.5967, "F1_UP_FORT": 0.3575, "F1_DOWN_FORT": 0.4167, "train_start": "2001-02-20", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VIX_Price_zscore_60d__div__hmm_p_stress", "vix_zscore_10d__div__heston_var_ev_h1", "NFCI_ret_5d__zrel__NEE_NextEra_ret_5d", "VIX_Price_zscore_60d__zrel__heston_var_ev_h3", "HangSeng_HK_ret_5d", "VIX_Price_zscore_60d__ret5x__COF_CapitalOne_ret_5d", "DUK_Duke_ret_5d__div__COF_CapitalOne_ret_5d", "DUK_Duke_ret_5d__ret5x__COF_CapitalOne_ret_5d", "EWL_Switzerland_vol_20d", "EWM_Malaysia_vol_20d", "EOG_EOGResources_vol_20d", "kalman_filtered__prod__VOD_Vodafone_vol_20d", "MSFT_ret_5d__div__NEE_NextEra_ret_5d", "GD_GeneralDynamics_zscore_60d", "ENB_EnbridgeInc_ret_1d", "INTC_ret_5d", "MSFT_ret_5d__zrel__AVB_AvalonBay_ret_5d", "VIX_Price_zscore_60d__div__heston_var_ev_h3", "US3M_Rate_zscore_60d", "kalman_filtered__minus__XLY_Disc_zscore_60d"], "is_new": false}, {"model_id": "v2_h3_NORMAL_GradientBoosting_N21", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 3, "n_features": 21, "F1_dir": 0.6038, "F1_UP_FORT": 0.3397, "F1_DOWN_FORT": 0.4051, "train_start": "2001-02-20", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VIX_Price_zscore_60d__div__hmm_p_stress", "vix_zscore_10d__div__heston_var_ev_h1", "NFCI_ret_5d__zrel__NEE_NextEra_ret_5d", "VIX_Price_zscore_60d__zrel__heston_var_ev_h3", "HangSeng_HK_ret_5d", "VIX_Price_zscore_60d__ret5x__COF_CapitalOne_ret_5d", "DUK_Duke_ret_5d__div__COF_CapitalOne_ret_5d", "DUK_Duke_ret_5d__ret5x__COF_CapitalOne_ret_5d", "EWL_Switzerland_vol_20d", "EWM_Malaysia_vol_20d", "EOG_EOGResources_vol_20d", "kalman_filtered__prod__VOD_Vodafone_vol_20d", "MSFT_ret_5d__div__NEE_NextEra_ret_5d", "GD_GeneralDynamics_zscore_60d", "ENB_EnbridgeInc_ret_1d", "INTC_ret_5d", "MSFT_ret_5d__zrel__AVB_AvalonBay_ret_5d", "VIX_Price_zscore_60d__div__heston_var_ev_h3", "US3M_Rate_zscore_60d", "kalman_filtered__minus__XLY_Disc_zscore_60d", "vix_zscore_10d__macross__MSFT_ret_5d"], "is_new": false}, {"model_id": "v2_h3_NORMAL_GradientBoosting_N22", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 3, "n_features": 22, "F1_dir": 0.6082, "F1_UP_FORT": 0.3558, "F1_DOWN_FORT": 0.4298, "train_start": "2001-02-20", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VIX_Price_zscore_60d__div__hmm_p_stress", "vix_zscore_10d__div__heston_var_ev_h1", "NFCI_ret_5d__zrel__NEE_NextEra_ret_5d", "VIX_Price_zscore_60d__zrel__heston_var_ev_h3", "HangSeng_HK_ret_5d", "VIX_Price_zscore_60d__ret5x__COF_CapitalOne_ret_5d", "DUK_Duke_ret_5d__div__COF_CapitalOne_ret_5d", "DUK_Duke_ret_5d__ret5x__COF_CapitalOne_ret_5d", "EWL_Switzerland_vol_20d", "EWM_Malaysia_vol_20d", "EOG_EOGResources_vol_20d", "kalman_filtered__prod__VOD_Vodafone_vol_20d", "MSFT_ret_5d__div__NEE_NextEra_ret_5d", "GD_GeneralDynamics_zscore_60d", "ENB_EnbridgeInc_ret_1d", "INTC_ret_5d", "MSFT_ret_5d__zrel__AVB_AvalonBay_ret_5d", "VIX_Price_zscore_60d__div__heston_var_ev_h3", "US3M_Rate_zscore_60d", "kalman_filtered__minus__XLY_Disc_zscore_60d", "vix_zscore_10d__macross__MSFT_ret_5d", "VOD_Vodafone_vol_20d__macross__COF_CapitalOne_ret_5d"], "is_new": false}, {"model_id": "v2_h3_NORMAL_RandomForest_N5", "algo": "RandomForest", "regime": "NORMAL", "horizon": 3, "n_features": 5, "F1_dir": 0.5362, "F1_UP_FORT": 0.3229, "F1_DOWN_FORT": 0.3542, "train_start": "2001-02-20", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VIX_Price_zscore_60d__div__hmm_p_stress", "vix_zscore_10d__div__heston_var_ev_h1", "NFCI_ret_5d__zrel__NEE_NextEra_ret_5d", "VIX_Price_zscore_60d__zrel__heston_var_ev_h3", "HangSeng_HK_ret_5d"], "is_new": false}, {"model_id": "v2_h3_NORMAL_RandomForest_N6", "algo": "RandomForest", "regime": "NORMAL", "horizon": 3, "n_features": 6, "F1_dir": 0.5532, "F1_UP_FORT": 0.3117, "F1_DOWN_FORT": 0.3696, "train_start": "2001-02-20", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VIX_Price_zscore_60d__div__hmm_p_stress", "vix_zscore_10d__div__heston_var_ev_h1", "NFCI_ret_5d__zrel__NEE_NextEra_ret_5d", "VIX_Price_zscore_60d__zrel__heston_var_ev_h3", "HangSeng_HK_ret_5d", "VIX_Price_zscore_60d__ret5x__COF_CapitalOne_ret_5d"], "is_new": false}, {"model_id": "v2_h3_NORMAL_RandomForest_N7", "algo": "RandomForest", "regime": "NORMAL", "horizon": 3, "n_features": 7, "F1_dir": 0.5475, "F1_UP_FORT": 0.2994, "F1_DOWN_FORT": 0.3761, "train_start": "2001-02-20", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VIX_Price_zscore_60d__div__hmm_p_stress", "vix_zscore_10d__div__heston_var_ev_h1", "NFCI_ret_5d__zrel__NEE_NextEra_ret_5d", "VIX_Price_zscore_60d__zrel__heston_var_ev_h3", "HangSeng_HK_ret_5d", "VIX_Price_zscore_60d__ret5x__COF_CapitalOne_ret_5d", "DUK_Duke_ret_5d__div__COF_CapitalOne_ret_5d"], "is_new": false}, {"model_id": "v2_h3_NORMAL_RandomForest_N8", "algo": "RandomForest", "regime": "NORMAL", "horizon": 3, "n_features": 8, "F1_dir": 0.5635, "F1_UP_FORT": 0.328, "F1_DOWN_FORT": 0.3919, "train_start": "2001-02-20", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VIX_Price_zscore_60d__div__hmm_p_stress", "vix_zscore_10d__div__heston_var_ev_h1", "NFCI_ret_5d__zrel__NEE_NextEra_ret_5d", "VIX_Price_zscore_60d__zrel__heston_var_ev_h3", "HangSeng_HK_ret_5d", "VIX_Price_zscore_60d__ret5x__COF_CapitalOne_ret_5d", "DUK_Duke_ret_5d__div__COF_CapitalOne_ret_5d", "DUK_Duke_ret_5d__ret5x__COF_CapitalOne_ret_5d"], "is_new": false}, {"model_id": "v2_h3_NORMAL_RandomForest_N9", "algo": "RandomForest", "regime": "NORMAL", "horizon": 3, "n_features": 9, "F1_dir": 0.5596, "F1_UP_FORT": 0.3055, "F1_DOWN_FORT": 0.384, "train_start": "2001-02-20", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VIX_Price_zscore_60d__div__hmm_p_stress", "vix_zscore_10d__div__heston_var_ev_h1", "NFCI_ret_5d__zrel__NEE_NextEra_ret_5d", "VIX_Price_zscore_60d__zrel__heston_var_ev_h3", "HangSeng_HK_ret_5d", "VIX_Price_zscore_60d__ret5x__COF_CapitalOne_ret_5d", "DUK_Duke_ret_5d__div__COF_CapitalOne_ret_5d", "DUK_Duke_ret_5d__ret5x__COF_CapitalOne_ret_5d", "EWL_Switzerland_vol_20d"], "is_new": false}, {"model_id": "v2_h3_NORMAL_RandomForest_N10", "algo": "RandomForest", "regime": "NORMAL", "horizon": 3, "n_features": 10, "F1_dir": 0.5497, "F1_UP_FORT": 0.3037, "F1_DOWN_FORT": 0.393, "train_start": "2001-02-20", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VIX_Price_zscore_60d__div__hmm_p_stress", "vix_zscore_10d__div__heston_var_ev_h1", "NFCI_ret_5d__zrel__NEE_NextEra_ret_5d", "VIX_Price_zscore_60d__zrel__heston_var_ev_h3", "HangSeng_HK_ret_5d", "VIX_Price_zscore_60d__ret5x__COF_CapitalOne_ret_5d", "DUK_Duke_ret_5d__div__COF_CapitalOne_ret_5d", "DUK_Duke_ret_5d__ret5x__COF_CapitalOne_ret_5d", "EWL_Switzerland_vol_20d", "EWM_Malaysia_vol_20d"], "is_new": false}, {"model_id": "v2_h3_NORMAL_RandomForest_N11", "algo": "RandomForest", "regime": "NORMAL", "horizon": 3, "n_features": 11, "F1_dir": 0.5645, "F1_UP_FORT": 0.3407, "F1_DOWN_FORT": 0.3947, "train_start": "2001-02-20", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VIX_Price_zscore_60d__div__hmm_p_stress", "vix_zscore_10d__div__heston_var_ev_h1", "NFCI_ret_5d__zrel__NEE_NextEra_ret_5d", "VIX_Price_zscore_60d__zrel__heston_var_ev_h3", "HangSeng_HK_ret_5d", "VIX_Price_zscore_60d__ret5x__COF_CapitalOne_ret_5d", "DUK_Duke_ret_5d__div__COF_CapitalOne_ret_5d", "DUK_Duke_ret_5d__ret5x__COF_CapitalOne_ret_5d", "EWL_Switzerland_vol_20d", "EWM_Malaysia_vol_20d", "EOG_EOGResources_vol_20d"], "is_new": false}, {"model_id": "v2_h3_NORMAL_RandomForest_N12", "algo": "RandomForest", "regime": "NORMAL", "horizon": 3, "n_features": 12, "F1_dir": 0.5762, "F1_UP_FORT": 0.3526, "F1_DOWN_FORT": 0.3877, "train_start": "2001-02-20", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VIX_Price_zscore_60d__div__hmm_p_stress", "vix_zscore_10d__div__heston_var_ev_h1", "NFCI_ret_5d__zrel__NEE_NextEra_ret_5d", "VIX_Price_zscore_60d__zrel__heston_var_ev_h3", "HangSeng_HK_ret_5d", "VIX_Price_zscore_60d__ret5x__COF_CapitalOne_ret_5d", "DUK_Duke_ret_5d__div__COF_CapitalOne_ret_5d", "DUK_Duke_ret_5d__ret5x__COF_CapitalOne_ret_5d", "EWL_Switzerland_vol_20d", "EWM_Malaysia_vol_20d", "EOG_EOGResources_vol_20d", "kalman_filtered__prod__VOD_Vodafone_vol_20d"], "is_new": false}, {"model_id": "v2_h3_NORMAL_RandomForest_N13", "algo": "RandomForest", "regime": "NORMAL", "horizon": 3, "n_features": 13, "F1_dir": 0.5623, "F1_UP_FORT": 0.3095, "F1_DOWN_FORT": 0.3878, "train_start": "2001-02-20", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VIX_Price_zscore_60d__div__hmm_p_stress", "vix_zscore_10d__div__heston_var_ev_h1", "NFCI_ret_5d__zrel__NEE_NextEra_ret_5d", "VIX_Price_zscore_60d__zrel__heston_var_ev_h3", "HangSeng_HK_ret_5d", "VIX_Price_zscore_60d__ret5x__COF_CapitalOne_ret_5d", "DUK_Duke_ret_5d__div__COF_CapitalOne_ret_5d", "DUK_Duke_ret_5d__ret5x__COF_CapitalOne_ret_5d", "EWL_Switzerland_vol_20d", "EWM_Malaysia_vol_20d", "EOG_EOGResources_vol_20d", "kalman_filtered__prod__VOD_Vodafone_vol_20d", "MSFT_ret_5d__div__NEE_NextEra_ret_5d"], "is_new": false}, {"model_id": "v2_h3_NORMAL_RandomForest_N14", "algo": "RandomForest", "regime": "NORMAL", "horizon": 3, "n_features": 14, "F1_dir": 0.5627, "F1_UP_FORT": 0.3351, "F1_DOWN_FORT": 0.4048, "train_start": "2001-02-20", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VIX_Price_zscore_60d__div__hmm_p_stress", "vix_zscore_10d__div__heston_var_ev_h1", "NFCI_ret_5d__zrel__NEE_NextEra_ret_5d", "VIX_Price_zscore_60d__zrel__heston_var_ev_h3", "HangSeng_HK_ret_5d", "VIX_Price_zscore_60d__ret5x__COF_CapitalOne_ret_5d", "DUK_Duke_ret_5d__div__COF_CapitalOne_ret_5d", "DUK_Duke_ret_5d__ret5x__COF_CapitalOne_ret_5d", "EWL_Switzerland_vol_20d", "EWM_Malaysia_vol_20d", "EOG_EOGResources_vol_20d", "kalman_filtered__prod__VOD_Vodafone_vol_20d", "MSFT_ret_5d__div__NEE_NextEra_ret_5d", "GD_GeneralDynamics_zscore_60d"], "is_new": false}, {"model_id": "v2_h3_NORMAL_RandomForest_N15", "algo": "RandomForest", "regime": "NORMAL", "horizon": 3, "n_features": 15, "F1_dir": 0.5758, "F1_UP_FORT": 0.3226, "F1_DOWN_FORT": 0.409, "train_start": "2001-02-20", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VIX_Price_zscore_60d__div__hmm_p_stress", "vix_zscore_10d__div__heston_var_ev_h1", "NFCI_ret_5d__zrel__NEE_NextEra_ret_5d", "VIX_Price_zscore_60d__zrel__heston_var_ev_h3", "HangSeng_HK_ret_5d", "VIX_Price_zscore_60d__ret5x__COF_CapitalOne_ret_5d", "DUK_Duke_ret_5d__div__COF_CapitalOne_ret_5d", "DUK_Duke_ret_5d__ret5x__COF_CapitalOne_ret_5d", "EWL_Switzerland_vol_20d", "EWM_Malaysia_vol_20d", "EOG_EOGResources_vol_20d", "kalman_filtered__prod__VOD_Vodafone_vol_20d", "MSFT_ret_5d__div__NEE_NextEra_ret_5d", "GD_GeneralDynamics_zscore_60d", "ENB_EnbridgeInc_ret_1d"], "is_new": false}, {"model_id": "v2_h3_NORMAL_RandomForest_N16", "algo": "RandomForest", "regime": "NORMAL", "horizon": 3, "n_features": 16, "F1_dir": 0.5618, "F1_UP_FORT": 0.3298, "F1_DOWN_FORT": 0.4041, "train_start": "2001-02-20", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VIX_Price_zscore_60d__div__hmm_p_stress", "vix_zscore_10d__div__heston_var_ev_h1", "NFCI_ret_5d__zrel__NEE_NextEra_ret_5d", "VIX_Price_zscore_60d__zrel__heston_var_ev_h3", "HangSeng_HK_ret_5d", "VIX_Price_zscore_60d__ret5x__COF_CapitalOne_ret_5d", "DUK_Duke_ret_5d__div__COF_CapitalOne_ret_5d", "DUK_Duke_ret_5d__ret5x__COF_CapitalOne_ret_5d", "EWL_Switzerland_vol_20d", "EWM_Malaysia_vol_20d", "EOG_EOGResources_vol_20d", "kalman_filtered__prod__VOD_Vodafone_vol_20d", "MSFT_ret_5d__div__NEE_NextEra_ret_5d", "GD_GeneralDynamics_zscore_60d", "ENB_EnbridgeInc_ret_1d", "INTC_ret_5d"], "is_new": false}, {"model_id": "v2_h3_NORMAL_RandomForest_N17", "algo": "RandomForest", "regime": "NORMAL", "horizon": 3, "n_features": 17, "F1_dir": 0.5822, "F1_UP_FORT": 0.3416, "F1_DOWN_FORT": 0.396, "train_start": "2001-02-20", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VIX_Price_zscore_60d__div__hmm_p_stress", "vix_zscore_10d__div__heston_var_ev_h1", "NFCI_ret_5d__zrel__NEE_NextEra_ret_5d", "VIX_Price_zscore_60d__zrel__heston_var_ev_h3", "HangSeng_HK_ret_5d", "VIX_Price_zscore_60d__ret5x__COF_CapitalOne_ret_5d", "DUK_Duke_ret_5d__div__COF_CapitalOne_ret_5d", "DUK_Duke_ret_5d__ret5x__COF_CapitalOne_ret_5d", "EWL_Switzerland_vol_20d", "EWM_Malaysia_vol_20d", "EOG_EOGResources_vol_20d", "kalman_filtered__prod__VOD_Vodafone_vol_20d", "MSFT_ret_5d__div__NEE_NextEra_ret_5d", "GD_GeneralDynamics_zscore_60d", "ENB_EnbridgeInc_ret_1d", "INTC_ret_5d", "MSFT_ret_5d__zrel__AVB_AvalonBay_ret_5d"], "is_new": false}, {"model_id": "v2_h3_NORMAL_RandomForest_N18", "algo": "RandomForest", "regime": "NORMAL", "horizon": 3, "n_features": 18, "F1_dir": 0.5701, "F1_UP_FORT": 0.3342, "F1_DOWN_FORT": 0.4206, "train_start": "2001-02-20", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VIX_Price_zscore_60d__div__hmm_p_stress", "vix_zscore_10d__div__heston_var_ev_h1", "NFCI_ret_5d__zrel__NEE_NextEra_ret_5d", "VIX_Price_zscore_60d__zrel__heston_var_ev_h3", "HangSeng_HK_ret_5d", "VIX_Price_zscore_60d__ret5x__COF_CapitalOne_ret_5d", "DUK_Duke_ret_5d__div__COF_CapitalOne_ret_5d", "DUK_Duke_ret_5d__ret5x__COF_CapitalOne_ret_5d", "EWL_Switzerland_vol_20d", "EWM_Malaysia_vol_20d", "EOG_EOGResources_vol_20d", "kalman_filtered__prod__VOD_Vodafone_vol_20d", "MSFT_ret_5d__div__NEE_NextEra_ret_5d", "GD_GeneralDynamics_zscore_60d", "ENB_EnbridgeInc_ret_1d", "INTC_ret_5d", "MSFT_ret_5d__zrel__AVB_AvalonBay_ret_5d", "VIX_Price_zscore_60d__div__heston_var_ev_h3"], "is_new": false}, {"model_id": "v2_h3_NORMAL_RandomForest_N19", "algo": "RandomForest", "regime": "NORMAL", "horizon": 3, "n_features": 19, "F1_dir": 0.5585, "F1_UP_FORT": 0.3116, "F1_DOWN_FORT": 0.4165, "train_start": "2001-02-20", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VIX_Price_zscore_60d__div__hmm_p_stress", "vix_zscore_10d__div__heston_var_ev_h1", "NFCI_ret_5d__zrel__NEE_NextEra_ret_5d", "VIX_Price_zscore_60d__zrel__heston_var_ev_h3", "HangSeng_HK_ret_5d", "VIX_Price_zscore_60d__ret5x__COF_CapitalOne_ret_5d", "DUK_Duke_ret_5d__div__COF_CapitalOne_ret_5d", "DUK_Duke_ret_5d__ret5x__COF_CapitalOne_ret_5d", "EWL_Switzerland_vol_20d", "EWM_Malaysia_vol_20d", "EOG_EOGResources_vol_20d", "kalman_filtered__prod__VOD_Vodafone_vol_20d", "MSFT_ret_5d__div__NEE_NextEra_ret_5d", "GD_GeneralDynamics_zscore_60d", "ENB_EnbridgeInc_ret_1d", "INTC_ret_5d", "MSFT_ret_5d__zrel__AVB_AvalonBay_ret_5d", "VIX_Price_zscore_60d__div__heston_var_ev_h3", "US3M_Rate_zscore_60d"], "is_new": false}, {"model_id": "v2_h3_NORMAL_RandomForest_N20", "algo": "RandomForest", "regime": "NORMAL", "horizon": 3, "n_features": 20, "F1_dir": 0.5604, "F1_UP_FORT": 0.314, "F1_DOWN_FORT": 0.4311, "train_start": "2001-02-20", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VIX_Price_zscore_60d__div__hmm_p_stress", "vix_zscore_10d__div__heston_var_ev_h1", "NFCI_ret_5d__zrel__NEE_NextEra_ret_5d", "VIX_Price_zscore_60d__zrel__heston_var_ev_h3", "HangSeng_HK_ret_5d", "VIX_Price_zscore_60d__ret5x__COF_CapitalOne_ret_5d", "DUK_Duke_ret_5d__div__COF_CapitalOne_ret_5d", "DUK_Duke_ret_5d__ret5x__COF_CapitalOne_ret_5d", "EWL_Switzerland_vol_20d", "EWM_Malaysia_vol_20d", "EOG_EOGResources_vol_20d", "kalman_filtered__prod__VOD_Vodafone_vol_20d", "MSFT_ret_5d__div__NEE_NextEra_ret_5d", "GD_GeneralDynamics_zscore_60d", "ENB_EnbridgeInc_ret_1d", "INTC_ret_5d", "MSFT_ret_5d__zrel__AVB_AvalonBay_ret_5d", "VIX_Price_zscore_60d__div__heston_var_ev_h3", "US3M_Rate_zscore_60d", "kalman_filtered__minus__XLY_Disc_zscore_60d"], "is_new": false}, {"model_id": "v2_h3_NORMAL_RandomForest_N21", "algo": "RandomForest", "regime": "NORMAL", "horizon": 3, "n_features": 21, "F1_dir": 0.5712, "F1_UP_FORT": 0.3218, "F1_DOWN_FORT": 0.4355, "train_start": "2001-02-20", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VIX_Price_zscore_60d__div__hmm_p_stress", "vix_zscore_10d__div__heston_var_ev_h1", "NFCI_ret_5d__zrel__NEE_NextEra_ret_5d", "VIX_Price_zscore_60d__zrel__heston_var_ev_h3", "HangSeng_HK_ret_5d", "VIX_Price_zscore_60d__ret5x__COF_CapitalOne_ret_5d", "DUK_Duke_ret_5d__div__COF_CapitalOne_ret_5d", "DUK_Duke_ret_5d__ret5x__COF_CapitalOne_ret_5d", "EWL_Switzerland_vol_20d", "EWM_Malaysia_vol_20d", "EOG_EOGResources_vol_20d", "kalman_filtered__prod__VOD_Vodafone_vol_20d", "MSFT_ret_5d__div__NEE_NextEra_ret_5d", "GD_GeneralDynamics_zscore_60d", "ENB_EnbridgeInc_ret_1d", "INTC_ret_5d", "MSFT_ret_5d__zrel__AVB_AvalonBay_ret_5d", "VIX_Price_zscore_60d__div__heston_var_ev_h3", "US3M_Rate_zscore_60d", "kalman_filtered__minus__XLY_Disc_zscore_60d", "vix_zscore_10d__macross__MSFT_ret_5d"], "is_new": false}, {"model_id": "v2_h3_NORMAL_RandomForest_N22", "algo": "RandomForest", "regime": "NORMAL", "horizon": 3, "n_features": 22, "F1_dir": 0.564, "F1_UP_FORT": 0.317, "F1_DOWN_FORT": 0.4181, "train_start": "2001-02-20", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VIX_Price_zscore_60d__div__hmm_p_stress", "vix_zscore_10d__div__heston_var_ev_h1", "NFCI_ret_5d__zrel__NEE_NextEra_ret_5d", "VIX_Price_zscore_60d__zrel__heston_var_ev_h3", "HangSeng_HK_ret_5d", "VIX_Price_zscore_60d__ret5x__COF_CapitalOne_ret_5d", "DUK_Duke_ret_5d__div__COF_CapitalOne_ret_5d", "DUK_Duke_ret_5d__ret5x__COF_CapitalOne_ret_5d", "EWL_Switzerland_vol_20d", "EWM_Malaysia_vol_20d", "EOG_EOGResources_vol_20d", "kalman_filtered__prod__VOD_Vodafone_vol_20d", "MSFT_ret_5d__div__NEE_NextEra_ret_5d", "GD_GeneralDynamics_zscore_60d", "ENB_EnbridgeInc_ret_1d", "INTC_ret_5d", "MSFT_ret_5d__zrel__AVB_AvalonBay_ret_5d", "VIX_Price_zscore_60d__div__heston_var_ev_h3", "US3M_Rate_zscore_60d", "kalman_filtered__minus__XLY_Disc_zscore_60d", "vix_zscore_10d__macross__MSFT_ret_5d", "VOD_Vodafone_vol_20d__macross__COF_CapitalOne_ret_5d"], "is_new": false}, {"model_id": "v2_h3_NORMAL_LogisticRegression_N5", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 3, "n_features": 5, "F1_dir": 0.5548, "F1_UP_FORT": 0.3789, "F1_DOWN_FORT": 0.404, "train_start": "2001-02-20", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VIX_Price_zscore_60d__div__hmm_p_stress", "vix_zscore_10d__div__heston_var_ev_h1", "NFCI_ret_5d__zrel__NEE_NextEra_ret_5d", "VIX_Price_zscore_60d__zrel__heston_var_ev_h3", "HangSeng_HK_ret_5d"], "is_new": false}, {"model_id": "v2_h3_NORMAL_LogisticRegression_N6", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 3, "n_features": 6, "F1_dir": 0.552, "F1_UP_FORT": 0.3729, "F1_DOWN_FORT": 0.4131, "train_start": "2001-02-20", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VIX_Price_zscore_60d__div__hmm_p_stress", "vix_zscore_10d__div__heston_var_ev_h1", "NFCI_ret_5d__zrel__NEE_NextEra_ret_5d", "VIX_Price_zscore_60d__zrel__heston_var_ev_h3", "HangSeng_HK_ret_5d", "VIX_Price_zscore_60d__ret5x__COF_CapitalOne_ret_5d"], "is_new": false}, {"model_id": "v2_h3_NORMAL_LogisticRegression_N7", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 3, "n_features": 7, "F1_dir": 0.5409, "F1_UP_FORT": 0.3697, "F1_DOWN_FORT": 0.4152, "train_start": "2001-02-20", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VIX_Price_zscore_60d__div__hmm_p_stress", "vix_zscore_10d__div__heston_var_ev_h1", "NFCI_ret_5d__zrel__NEE_NextEra_ret_5d", "VIX_Price_zscore_60d__zrel__heston_var_ev_h3", "HangSeng_HK_ret_5d", "VIX_Price_zscore_60d__ret5x__COF_CapitalOne_ret_5d", "DUK_Duke_ret_5d__div__COF_CapitalOne_ret_5d"], "is_new": false}, {"model_id": "v2_h3_NORMAL_LogisticRegression_N8", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 3, "n_features": 8, "F1_dir": 0.5368, "F1_UP_FORT": 0.3723, "F1_DOWN_FORT": 0.4031, "train_start": "2001-02-20", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VIX_Price_zscore_60d__div__hmm_p_stress", "vix_zscore_10d__div__heston_var_ev_h1", "NFCI_ret_5d__zrel__NEE_NextEra_ret_5d", "VIX_Price_zscore_60d__zrel__heston_var_ev_h3", "HangSeng_HK_ret_5d", "VIX_Price_zscore_60d__ret5x__COF_CapitalOne_ret_5d", "DUK_Duke_ret_5d__div__COF_CapitalOne_ret_5d", "DUK_Duke_ret_5d__ret5x__COF_CapitalOne_ret_5d"], "is_new": false}, {"model_id": "v2_h3_NORMAL_LogisticRegression_N9", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 3, "n_features": 9, "F1_dir": 0.5512, "F1_UP_FORT": 0.42, "F1_DOWN_FORT": 0.4115, "train_start": "2001-02-20", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VIX_Price_zscore_60d__div__hmm_p_stress", "vix_zscore_10d__div__heston_var_ev_h1", "NFCI_ret_5d__zrel__NEE_NextEra_ret_5d", "VIX_Price_zscore_60d__zrel__heston_var_ev_h3", "HangSeng_HK_ret_5d", "VIX_Price_zscore_60d__ret5x__COF_CapitalOne_ret_5d", "DUK_Duke_ret_5d__div__COF_CapitalOne_ret_5d", "DUK_Duke_ret_5d__ret5x__COF_CapitalOne_ret_5d", "EWL_Switzerland_vol_20d"], "is_new": false}, {"model_id": "v2_h3_NORMAL_LogisticRegression_N10", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 3, "n_features": 10, "F1_dir": 0.5596, "F1_UP_FORT": 0.4337, "F1_DOWN_FORT": 0.3947, "train_start": "2001-02-20", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VIX_Price_zscore_60d__div__hmm_p_stress", "vix_zscore_10d__div__heston_var_ev_h1", "NFCI_ret_5d__zrel__NEE_NextEra_ret_5d", "VIX_Price_zscore_60d__zrel__heston_var_ev_h3", "HangSeng_HK_ret_5d", "VIX_Price_zscore_60d__ret5x__COF_CapitalOne_ret_5d", "DUK_Duke_ret_5d__div__COF_CapitalOne_ret_5d", "DUK_Duke_ret_5d__ret5x__COF_CapitalOne_ret_5d", "EWL_Switzerland_vol_20d", "EWM_Malaysia_vol_20d"], "is_new": false}, {"model_id": "v2_h3_NORMAL_LogisticRegression_N11", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 3, "n_features": 11, "F1_dir": 0.5572, "F1_UP_FORT": 0.4055, "F1_DOWN_FORT": 0.352, "train_start": "2001-02-20", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VIX_Price_zscore_60d__div__hmm_p_stress", "vix_zscore_10d__div__heston_var_ev_h1", "NFCI_ret_5d__zrel__NEE_NextEra_ret_5d", "VIX_Price_zscore_60d__zrel__heston_var_ev_h3", "HangSeng_HK_ret_5d", "VIX_Price_zscore_60d__ret5x__COF_CapitalOne_ret_5d", "DUK_Duke_ret_5d__div__COF_CapitalOne_ret_5d", "DUK_Duke_ret_5d__ret5x__COF_CapitalOne_ret_5d", "EWL_Switzerland_vol_20d", "EWM_Malaysia_vol_20d", "EOG_EOGResources_vol_20d"], "is_new": false}, {"model_id": "v2_h3_NORMAL_LogisticRegression_N12", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 3, "n_features": 12, "F1_dir": 0.5407, "F1_UP_FORT": 0.4, "F1_DOWN_FORT": 0.3353, "train_start": "2001-02-20", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VIX_Price_zscore_60d__div__hmm_p_stress", "vix_zscore_10d__div__heston_var_ev_h1", "NFCI_ret_5d__zrel__NEE_NextEra_ret_5d", "VIX_Price_zscore_60d__zrel__heston_var_ev_h3", "HangSeng_HK_ret_5d", "VIX_Price_zscore_60d__ret5x__COF_CapitalOne_ret_5d", "DUK_Duke_ret_5d__div__COF_CapitalOne_ret_5d", "DUK_Duke_ret_5d__ret5x__COF_CapitalOne_ret_5d", "EWL_Switzerland_vol_20d", "EWM_Malaysia_vol_20d", "EOG_EOGResources_vol_20d", "kalman_filtered__prod__VOD_Vodafone_vol_20d"], "is_new": false}, {"model_id": "v2_h3_NORMAL_LogisticRegression_N13", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 3, "n_features": 13, "F1_dir": 0.5378, "F1_UP_FORT": 0.3909, "F1_DOWN_FORT": 0.3235, "train_start": "2001-02-20", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VIX_Price_zscore_60d__div__hmm_p_stress", "vix_zscore_10d__div__heston_var_ev_h1", "NFCI_ret_5d__zrel__NEE_NextEra_ret_5d", "VIX_Price_zscore_60d__zrel__heston_var_ev_h3", "HangSeng_HK_ret_5d", "VIX_Price_zscore_60d__ret5x__COF_CapitalOne_ret_5d", "DUK_Duke_ret_5d__div__COF_CapitalOne_ret_5d", "DUK_Duke_ret_5d__ret5x__COF_CapitalOne_ret_5d", "EWL_Switzerland_vol_20d", "EWM_Malaysia_vol_20d", "EOG_EOGResources_vol_20d", "kalman_filtered__prod__VOD_Vodafone_vol_20d", "MSFT_ret_5d__div__NEE_NextEra_ret_5d"], "is_new": false}, {"model_id": "v2_h3_NORMAL_LogisticRegression_N14", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 3, "n_features": 14, "F1_dir": 0.5436, "F1_UP_FORT": 0.386, "F1_DOWN_FORT": 0.3516, "train_start": "2001-02-20", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VIX_Price_zscore_60d__div__hmm_p_stress", "vix_zscore_10d__div__heston_var_ev_h1", "NFCI_ret_5d__zrel__NEE_NextEra_ret_5d", "VIX_Price_zscore_60d__zrel__heston_var_ev_h3", "HangSeng_HK_ret_5d", "VIX_Price_zscore_60d__ret5x__COF_CapitalOne_ret_5d", "DUK_Duke_ret_5d__div__COF_CapitalOne_ret_5d", "DUK_Duke_ret_5d__ret5x__COF_CapitalOne_ret_5d", "EWL_Switzerland_vol_20d", "EWM_Malaysia_vol_20d", "EOG_EOGResources_vol_20d", "kalman_filtered__prod__VOD_Vodafone_vol_20d", "MSFT_ret_5d__div__NEE_NextEra_ret_5d", "GD_GeneralDynamics_zscore_60d"], "is_new": false}, {"model_id": "v2_h3_NORMAL_LogisticRegression_N15", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 3, "n_features": 15, "F1_dir": 0.5532, "F1_UP_FORT": 0.3991, "F1_DOWN_FORT": 0.3516, "train_start": "2001-02-20", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VIX_Price_zscore_60d__div__hmm_p_stress", "vix_zscore_10d__div__heston_var_ev_h1", "NFCI_ret_5d__zrel__NEE_NextEra_ret_5d", "VIX_Price_zscore_60d__zrel__heston_var_ev_h3", "HangSeng_HK_ret_5d", "VIX_Price_zscore_60d__ret5x__COF_CapitalOne_ret_5d", "DUK_Duke_ret_5d__div__COF_CapitalOne_ret_5d", "DUK_Duke_ret_5d__ret5x__COF_CapitalOne_ret_5d", "EWL_Switzerland_vol_20d", "EWM_Malaysia_vol_20d", "EOG_EOGResources_vol_20d", "kalman_filtered__prod__VOD_Vodafone_vol_20d", "MSFT_ret_5d__div__NEE_NextEra_ret_5d", "GD_GeneralDynamics_zscore_60d", "ENB_EnbridgeInc_ret_1d"], "is_new": false}, {"model_id": "v2_h3_NORMAL_LogisticRegression_N16", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 3, "n_features": 16, "F1_dir": 0.5345, "F1_UP_FORT": 0.3498, "F1_DOWN_FORT": 0.3626, "train_start": "2001-02-20", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VIX_Price_zscore_60d__div__hmm_p_stress", "vix_zscore_10d__div__heston_var_ev_h1", "NFCI_ret_5d__zrel__NEE_NextEra_ret_5d", "VIX_Price_zscore_60d__zrel__heston_var_ev_h3", "HangSeng_HK_ret_5d", "VIX_Price_zscore_60d__ret5x__COF_CapitalOne_ret_5d", "DUK_Duke_ret_5d__div__COF_CapitalOne_ret_5d", "DUK_Duke_ret_5d__ret5x__COF_CapitalOne_ret_5d", "EWL_Switzerland_vol_20d", "EWM_Malaysia_vol_20d", "EOG_EOGResources_vol_20d", "kalman_filtered__prod__VOD_Vodafone_vol_20d", "MSFT_ret_5d__div__NEE_NextEra_ret_5d", "GD_GeneralDynamics_zscore_60d", "ENB_EnbridgeInc_ret_1d", "INTC_ret_5d"], "is_new": false}, {"model_id": "v2_h3_NORMAL_LogisticRegression_N17", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 3, "n_features": 17, "F1_dir": 0.5388, "F1_UP_FORT": 0.3685, "F1_DOWN_FORT": 0.3503, "train_start": "2001-02-20", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VIX_Price_zscore_60d__div__hmm_p_stress", "vix_zscore_10d__div__heston_var_ev_h1", "NFCI_ret_5d__zrel__NEE_NextEra_ret_5d", "VIX_Price_zscore_60d__zrel__heston_var_ev_h3", "HangSeng_HK_ret_5d", "VIX_Price_zscore_60d__ret5x__COF_CapitalOne_ret_5d", "DUK_Duke_ret_5d__div__COF_CapitalOne_ret_5d", "DUK_Duke_ret_5d__ret5x__COF_CapitalOne_ret_5d", "EWL_Switzerland_vol_20d", "EWM_Malaysia_vol_20d", "EOG_EOGResources_vol_20d", "kalman_filtered__prod__VOD_Vodafone_vol_20d", "MSFT_ret_5d__div__NEE_NextEra_ret_5d", "GD_GeneralDynamics_zscore_60d", "ENB_EnbridgeInc_ret_1d", "INTC_ret_5d", "MSFT_ret_5d__zrel__AVB_AvalonBay_ret_5d"], "is_new": false}, {"model_id": "v2_h3_NORMAL_LogisticRegression_N18", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 3, "n_features": 18, "F1_dir": 0.5436, "F1_UP_FORT": 0.391, "F1_DOWN_FORT": 0.3641, "train_start": "2001-02-20", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VIX_Price_zscore_60d__div__hmm_p_stress", "vix_zscore_10d__div__heston_var_ev_h1", "NFCI_ret_5d__zrel__NEE_NextEra_ret_5d", "VIX_Price_zscore_60d__zrel__heston_var_ev_h3", "HangSeng_HK_ret_5d", "VIX_Price_zscore_60d__ret5x__COF_CapitalOne_ret_5d", "DUK_Duke_ret_5d__div__COF_CapitalOne_ret_5d", "DUK_Duke_ret_5d__ret5x__COF_CapitalOne_ret_5d", "EWL_Switzerland_vol_20d", "EWM_Malaysia_vol_20d", "EOG_EOGResources_vol_20d", "kalman_filtered__prod__VOD_Vodafone_vol_20d", "MSFT_ret_5d__div__NEE_NextEra_ret_5d", "GD_GeneralDynamics_zscore_60d", "ENB_EnbridgeInc_ret_1d", "INTC_ret_5d", "MSFT_ret_5d__zrel__AVB_AvalonBay_ret_5d", "VIX_Price_zscore_60d__div__heston_var_ev_h3"], "is_new": false}, {"model_id": "v2_h3_NORMAL_LogisticRegression_N19", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 3, "n_features": 19, "F1_dir": 0.5381, "F1_UP_FORT": 0.3532, "F1_DOWN_FORT": 0.3914, "train_start": "2001-02-20", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VIX_Price_zscore_60d__div__hmm_p_stress", "vix_zscore_10d__div__heston_var_ev_h1", "NFCI_ret_5d__zrel__NEE_NextEra_ret_5d", "VIX_Price_zscore_60d__zrel__heston_var_ev_h3", "HangSeng_HK_ret_5d", "VIX_Price_zscore_60d__ret5x__COF_CapitalOne_ret_5d", "DUK_Duke_ret_5d__div__COF_CapitalOne_ret_5d", "DUK_Duke_ret_5d__ret5x__COF_CapitalOne_ret_5d", "EWL_Switzerland_vol_20d", "EWM_Malaysia_vol_20d", "EOG_EOGResources_vol_20d", "kalman_filtered__prod__VOD_Vodafone_vol_20d", "MSFT_ret_5d__div__NEE_NextEra_ret_5d", "GD_GeneralDynamics_zscore_60d", "ENB_EnbridgeInc_ret_1d", "INTC_ret_5d", "MSFT_ret_5d__zrel__AVB_AvalonBay_ret_5d", "VIX_Price_zscore_60d__div__heston_var_ev_h3", "US3M_Rate_zscore_60d"], "is_new": false}, {"model_id": "v2_h3_NORMAL_LogisticRegression_N20", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 3, "n_features": 20, "F1_dir": 0.5534, "F1_UP_FORT": 0.3632, "F1_DOWN_FORT": 0.4192, "train_start": "2001-02-20", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VIX_Price_zscore_60d__div__hmm_p_stress", "vix_zscore_10d__div__heston_var_ev_h1", "NFCI_ret_5d__zrel__NEE_NextEra_ret_5d", "VIX_Price_zscore_60d__zrel__heston_var_ev_h3", "HangSeng_HK_ret_5d", "VIX_Price_zscore_60d__ret5x__COF_CapitalOne_ret_5d", "DUK_Duke_ret_5d__div__COF_CapitalOne_ret_5d", "DUK_Duke_ret_5d__ret5x__COF_CapitalOne_ret_5d", "EWL_Switzerland_vol_20d", "EWM_Malaysia_vol_20d", "EOG_EOGResources_vol_20d", "kalman_filtered__prod__VOD_Vodafone_vol_20d", "MSFT_ret_5d__div__NEE_NextEra_ret_5d", "GD_GeneralDynamics_zscore_60d", "ENB_EnbridgeInc_ret_1d", "INTC_ret_5d", "MSFT_ret_5d__zrel__AVB_AvalonBay_ret_5d", "VIX_Price_zscore_60d__div__heston_var_ev_h3", "US3M_Rate_zscore_60d", "kalman_filtered__minus__XLY_Disc_zscore_60d"], "is_new": false}, {"model_id": "v2_h3_NORMAL_LogisticRegression_N21", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 3, "n_features": 21, "F1_dir": 0.5534, "F1_UP_FORT": 0.3632, "F1_DOWN_FORT": 0.414, "train_start": "2001-02-20", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VIX_Price_zscore_60d__div__hmm_p_stress", "vix_zscore_10d__div__heston_var_ev_h1", "NFCI_ret_5d__zrel__NEE_NextEra_ret_5d", "VIX_Price_zscore_60d__zrel__heston_var_ev_h3", "HangSeng_HK_ret_5d", "VIX_Price_zscore_60d__ret5x__COF_CapitalOne_ret_5d", "DUK_Duke_ret_5d__div__COF_CapitalOne_ret_5d", "DUK_Duke_ret_5d__ret5x__COF_CapitalOne_ret_5d", "EWL_Switzerland_vol_20d", "EWM_Malaysia_vol_20d", "EOG_EOGResources_vol_20d", "kalman_filtered__prod__VOD_Vodafone_vol_20d", "MSFT_ret_5d__div__NEE_NextEra_ret_5d", "GD_GeneralDynamics_zscore_60d", "ENB_EnbridgeInc_ret_1d", "INTC_ret_5d", "MSFT_ret_5d__zrel__AVB_AvalonBay_ret_5d", "VIX_Price_zscore_60d__div__heston_var_ev_h3", "US3M_Rate_zscore_60d", "kalman_filtered__minus__XLY_Disc_zscore_60d", "vix_zscore_10d__macross__MSFT_ret_5d"], "is_new": false}, {"model_id": "v2_h3_NORMAL_LogisticRegression_N22", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 3, "n_features": 22, "F1_dir": 0.5534, "F1_UP_FORT": 0.3632, "F1_DOWN_FORT": 0.4122, "train_start": "2001-02-20", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VIX_Price_zscore_60d__div__hmm_p_stress", "vix_zscore_10d__div__heston_var_ev_h1", "NFCI_ret_5d__zrel__NEE_NextEra_ret_5d", "VIX_Price_zscore_60d__zrel__heston_var_ev_h3", "HangSeng_HK_ret_5d", "VIX_Price_zscore_60d__ret5x__COF_CapitalOne_ret_5d", "DUK_Duke_ret_5d__div__COF_CapitalOne_ret_5d", "DUK_Duke_ret_5d__ret5x__COF_CapitalOne_ret_5d", "EWL_Switzerland_vol_20d", "EWM_Malaysia_vol_20d", "EOG_EOGResources_vol_20d", "kalman_filtered__prod__VOD_Vodafone_vol_20d", "MSFT_ret_5d__div__NEE_NextEra_ret_5d", "GD_GeneralDynamics_zscore_60d", "ENB_EnbridgeInc_ret_1d", "INTC_ret_5d", "MSFT_ret_5d__zrel__AVB_AvalonBay_ret_5d", "VIX_Price_zscore_60d__div__heston_var_ev_h3", "US3M_Rate_zscore_60d", "kalman_filtered__minus__XLY_Disc_zscore_60d", "vix_zscore_10d__macross__MSFT_ret_5d", "VOD_Vodafone_vol_20d__macross__COF_CapitalOne_ret_5d"], "is_new": false}, {"model_id": "v2_h3_NORMAL_GradientBoosting_Optuna_N17", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 3, "n_features": 17, "F1_dir": 0.5848, "F1_UP_FORT": 0.3627, "F1_DOWN_FORT": 0.383, "train_start": "2001-02-20", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VIX_Price_zscore_60d__div__hmm_p_stress", "vix_zscore_10d__div__heston_var_ev_h1", "NFCI_ret_5d__zrel__NEE_NextEra_ret_5d", "VIX_Price_zscore_60d__zrel__heston_var_ev_h3", "HangSeng_HK_ret_5d", "VIX_Price_zscore_60d__ret5x__COF_CapitalOne_ret_5d", "DUK_Duke_ret_5d__div__COF_CapitalOne_ret_5d", "DUK_Duke_ret_5d__ret5x__COF_CapitalOne_ret_5d", "EWL_Switzerland_vol_20d", "EWM_Malaysia_vol_20d", "EOG_EOGResources_vol_20d", "kalman_filtered__prod__VOD_Vodafone_vol_20d", "MSFT_ret_5d__div__NEE_NextEra_ret_5d", "GD_GeneralDynamics_zscore_60d", "ENB_EnbridgeInc_ret_1d", "INTC_ret_5d", "MSFT_ret_5d__zrel__AVB_AvalonBay_ret_5d"], "is_new": false}, {"model_id": "v2_h3_NORMAL_GradientBoosting_OptunaCal_N17", "algo": "GradientBoostingCal", "regime": "NORMAL", "horizon": 3, "n_features": 17, "F1_dir": 0.5584, "F1_UP_FORT": 0.1418, "F1_DOWN_FORT": 0.3878, "train_start": "2001-02-20", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VIX_Price_zscore_60d__div__hmm_p_stress", "vix_zscore_10d__div__heston_var_ev_h1", "NFCI_ret_5d__zrel__NEE_NextEra_ret_5d", "VIX_Price_zscore_60d__zrel__heston_var_ev_h3", "HangSeng_HK_ret_5d", "VIX_Price_zscore_60d__ret5x__COF_CapitalOne_ret_5d", "DUK_Duke_ret_5d__div__COF_CapitalOne_ret_5d", "DUK_Duke_ret_5d__ret5x__COF_CapitalOne_ret_5d", "EWL_Switzerland_vol_20d", "EWM_Malaysia_vol_20d", "EOG_EOGResources_vol_20d", "kalman_filtered__prod__VOD_Vodafone_vol_20d", "MSFT_ret_5d__div__NEE_NextEra_ret_5d", "GD_GeneralDynamics_zscore_60d", "ENB_EnbridgeInc_ret_1d", "INTC_ret_5d", "MSFT_ret_5d__zrel__AVB_AvalonBay_ret_5d"], "is_new": false}, {"model_id": "v2_h3_STRESS_XGBoost_N5", "algo": "XGBoost", "regime": "STRESS", "horizon": 3, "n_features": 5, "F1_dir": 0.5095, "F1_UP_FORT": 0.2889, "F1_DOWN_FORT": 0.4046, "train_start": "2000-11-15", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_mean_abs_ret_5d__prod__heston_xi", "Core_PCE_zscore_60d", "VIX_Price_zscore_60d__ret5x__MCD_ret_1d", "VIX_Price_zscore_60d__macross__AMD_zscore_60d", "ENB_EnbridgeInc_ret_5d__zrel__CMCSA_zscore_60d"], "is_new": false}, {"model_id": "v2_h3_STRESS_XGBoost_N6", "algo": "XGBoost", "regime": "STRESS", "horizon": 3, "n_features": 6, "F1_dir": 0.5068, "F1_UP_FORT": 0.1931, "F1_DOWN_FORT": 0.4769, "train_start": "2000-11-15", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_mean_abs_ret_5d__prod__heston_xi", "Core_PCE_zscore_60d", "VIX_Price_zscore_60d__ret5x__MCD_ret_1d", "VIX_Price_zscore_60d__macross__AMD_zscore_60d", "ENB_EnbridgeInc_ret_5d__zrel__CMCSA_zscore_60d", "US7Y_Rate_ret_20d"], "is_new": false}, {"model_id": "v2_h3_STRESS_XGBoost_N7", "algo": "XGBoost", "regime": "STRESS", "horizon": 3, "n_features": 7, "F1_dir": 0.513, "F1_UP_FORT": 0.1831, "F1_DOWN_FORT": 0.484, "train_start": "2000-11-15", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_mean_abs_ret_5d__prod__heston_xi", "Core_PCE_zscore_60d", "VIX_Price_zscore_60d__ret5x__MCD_ret_1d", "VIX_Price_zscore_60d__macross__AMD_zscore_60d", "ENB_EnbridgeInc_ret_5d__zrel__CMCSA_zscore_60d", "US7Y_Rate_ret_20d", "JNJ_ret_1d"], "is_new": false}, {"model_id": "v2_h3_STRESS_XGBoost_N10", "algo": "XGBoost", "regime": "STRESS", "horizon": 3, "n_features": 10, "F1_dir": 0.5108, "F1_UP_FORT": 0.2439, "F1_DOWN_FORT": 0.4361, "train_start": "2000-11-15", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_mean_abs_ret_5d__prod__heston_xi", "Core_PCE_zscore_60d", "VIX_Price_zscore_60d__ret5x__MCD_ret_1d", "VIX_Price_zscore_60d__macross__AMD_zscore_60d", "ENB_EnbridgeInc_ret_5d__zrel__CMCSA_zscore_60d", "US7Y_Rate_ret_20d", "JNJ_ret_1d", "EQIX_Equinix_ret_5d", "TM_Telephone_ret_1d", "CVX_ret_20d__zrel__NFCI_ret_5d"], "is_new": false}, {"model_id": "v2_h3_STRESS_XGBoost_N11", "algo": "XGBoost", "regime": "STRESS", "horizon": 3, "n_features": 11, "F1_dir": 0.5345, "F1_UP_FORT": 0.2767, "F1_DOWN_FORT": 0.4632, "train_start": "2000-11-15", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_mean_abs_ret_5d__prod__heston_xi", "Core_PCE_zscore_60d", "VIX_Price_zscore_60d__ret5x__MCD_ret_1d", "VIX_Price_zscore_60d__macross__AMD_zscore_60d", "ENB_EnbridgeInc_ret_5d__zrel__CMCSA_zscore_60d", "US7Y_Rate_ret_20d", "JNJ_ret_1d", "EQIX_Equinix_ret_5d", "TM_Telephone_ret_1d", "CVX_ret_20d__zrel__NFCI_ret_5d", "heston_xi__div__hmm_p_stress"], "is_new": false}, {"model_id": "v2_h3_STRESS_XGBoost_N12", "algo": "XGBoost", "regime": "STRESS", "horizon": 3, "n_features": 12, "F1_dir": 0.5281, "F1_UP_FORT": 0.2805, "F1_DOWN_FORT": 0.4818, "train_start": "2000-11-15", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_mean_abs_ret_5d__prod__heston_xi", "Core_PCE_zscore_60d", "VIX_Price_zscore_60d__ret5x__MCD_ret_1d", "VIX_Price_zscore_60d__macross__AMD_zscore_60d", "ENB_EnbridgeInc_ret_5d__zrel__CMCSA_zscore_60d", "US7Y_Rate_ret_20d", "JNJ_ret_1d", "EQIX_Equinix_ret_5d", "TM_Telephone_ret_1d", "CVX_ret_20d__zrel__NFCI_ret_5d", "heston_xi__div__hmm_p_stress", "ENB_EnbridgeInc_ret_5d__div__hmm_p_stress"], "is_new": false}, {"model_id": "v2_h3_STRESS_XGBoost_N13", "algo": "XGBoost", "regime": "STRESS", "horizon": 3, "n_features": 13, "F1_dir": 0.5484, "F1_UP_FORT": 0.3114, "F1_DOWN_FORT": 0.4873, "train_start": "2000-11-15", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_mean_abs_ret_5d__prod__heston_xi", "Core_PCE_zscore_60d", "VIX_Price_zscore_60d__ret5x__MCD_ret_1d", "VIX_Price_zscore_60d__macross__AMD_zscore_60d", "ENB_EnbridgeInc_ret_5d__zrel__CMCSA_zscore_60d", "US7Y_Rate_ret_20d", "JNJ_ret_1d", "EQIX_Equinix_ret_5d", "TM_Telephone_ret_1d", "CVX_ret_20d__zrel__NFCI_ret_5d", "heston_xi__div__hmm_p_stress", "ENB_EnbridgeInc_ret_5d__div__hmm_p_stress", "COST_ret_5d__div__vix_vs_ma20"], "is_new": false}, {"model_id": "v2_h3_STRESS_XGBoost_N14", "algo": "XGBoost", "regime": "STRESS", "horizon": 3, "n_features": 14, "F1_dir": 0.5613, "F1_UP_FORT": 0.3077, "F1_DOWN_FORT": 0.4765, "train_start": "2000-11-15", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_mean_abs_ret_5d__prod__heston_xi", "Core_PCE_zscore_60d", "VIX_Price_zscore_60d__ret5x__MCD_ret_1d", "VIX_Price_zscore_60d__macross__AMD_zscore_60d", "ENB_EnbridgeInc_ret_5d__zrel__CMCSA_zscore_60d", "US7Y_Rate_ret_20d", "JNJ_ret_1d", "EQIX_Equinix_ret_5d", "TM_Telephone_ret_1d", "CVX_ret_20d__zrel__NFCI_ret_5d", "heston_xi__div__hmm_p_stress", "ENB_EnbridgeInc_ret_5d__div__hmm_p_stress", "COST_ret_5d__div__vix_vs_ma20", "hmm_p_stress"], "is_new": false}, {"model_id": "v2_h3_STRESS_XGBoost_N15", "algo": "XGBoost", "regime": "STRESS", "horizon": 3, "n_features": 15, "F1_dir": 0.556, "F1_UP_FORT": 0.2917, "F1_DOWN_FORT": 0.4702, "train_start": "2000-11-15", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_mean_abs_ret_5d__prod__heston_xi", "Core_PCE_zscore_60d", "VIX_Price_zscore_60d__ret5x__MCD_ret_1d", "VIX_Price_zscore_60d__macross__AMD_zscore_60d", "ENB_EnbridgeInc_ret_5d__zrel__CMCSA_zscore_60d", "US7Y_Rate_ret_20d", "JNJ_ret_1d", "EQIX_Equinix_ret_5d", "TM_Telephone_ret_1d", "CVX_ret_20d__zrel__NFCI_ret_5d", "heston_xi__div__hmm_p_stress", "ENB_EnbridgeInc_ret_5d__div__hmm_p_stress", "COST_ret_5d__div__vix_vs_ma20", "hmm_p_stress", "PFE_ret_1d"], "is_new": false}, {"model_id": "v2_h3_STRESS_XGBoost_N16", "algo": "XGBoost", "regime": "STRESS", "horizon": 3, "n_features": 16, "F1_dir": 0.549, "F1_UP_FORT": 0.3097, "F1_DOWN_FORT": 0.4674, "train_start": "2000-11-15", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_mean_abs_ret_5d__prod__heston_xi", "Core_PCE_zscore_60d", "VIX_Price_zscore_60d__ret5x__MCD_ret_1d", "VIX_Price_zscore_60d__macross__AMD_zscore_60d", "ENB_EnbridgeInc_ret_5d__zrel__CMCSA_zscore_60d", "US7Y_Rate_ret_20d", "JNJ_ret_1d", "EQIX_Equinix_ret_5d", "TM_Telephone_ret_1d", "CVX_ret_20d__zrel__NFCI_ret_5d", "heston_xi__div__hmm_p_stress", "ENB_EnbridgeInc_ret_5d__div__hmm_p_stress", "COST_ret_5d__div__vix_vs_ma20", "hmm_p_stress", "PFE_ret_1d", "ENB_EnbridgeInc_ret_5d__macross__CMCSA_zscore_60d"], "is_new": false}, {"model_id": "v2_h3_STRESS_XGBoost_N17", "algo": "XGBoost", "regime": "STRESS", "horizon": 3, "n_features": 17, "F1_dir": 0.5536, "F1_UP_FORT": 0.3205, "F1_DOWN_FORT": 0.4795, "train_start": "2000-11-15", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_mean_abs_ret_5d__prod__heston_xi", "Core_PCE_zscore_60d", "VIX_Price_zscore_60d__ret5x__MCD_ret_1d", "VIX_Price_zscore_60d__macross__AMD_zscore_60d", "ENB_EnbridgeInc_ret_5d__zrel__CMCSA_zscore_60d", "US7Y_Rate_ret_20d", "JNJ_ret_1d", "EQIX_Equinix_ret_5d", "TM_Telephone_ret_1d", "CVX_ret_20d__zrel__NFCI_ret_5d", "heston_xi__div__hmm_p_stress", "ENB_EnbridgeInc_ret_5d__div__hmm_p_stress", "COST_ret_5d__div__vix_vs_ma20", "hmm_p_stress", "PFE_ret_1d", "ENB_EnbridgeInc_ret_5d__macross__CMCSA_zscore_60d", "VIX_Price_zscore_60d__ret5x__AMD_zscore_60d"], "is_new": false}, {"model_id": "v2_h3_STRESS_XGBoost_N18", "algo": "XGBoost", "regime": "STRESS", "horizon": 3, "n_features": 18, "F1_dir": 0.5345, "F1_UP_FORT": 0.28, "F1_DOWN_FORT": 0.4846, "train_start": "2000-11-15", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_mean_abs_ret_5d__prod__heston_xi", "Core_PCE_zscore_60d", "VIX_Price_zscore_60d__ret5x__MCD_ret_1d", "VIX_Price_zscore_60d__macross__AMD_zscore_60d", "ENB_EnbridgeInc_ret_5d__zrel__CMCSA_zscore_60d", "US7Y_Rate_ret_20d", "JNJ_ret_1d", "EQIX_Equinix_ret_5d", "TM_Telephone_ret_1d", "CVX_ret_20d__zrel__NFCI_ret_5d", "heston_xi__div__hmm_p_stress", "ENB_EnbridgeInc_ret_5d__div__hmm_p_stress", "COST_ret_5d__div__vix_vs_ma20", "hmm_p_stress", "PFE_ret_1d", "ENB_EnbridgeInc_ret_5d__macross__CMCSA_zscore_60d", "VIX_Price_zscore_60d__ret5x__AMD_zscore_60d", "ORCL_zscore_60d"], "is_new": false}, {"model_id": "v2_h3_STRESS_LightGBM_N10", "algo": "LightGBM", "regime": "STRESS", "horizon": 3, "n_features": 10, "F1_dir": 0.5362, "F1_UP_FORT": 0.236, "F1_DOWN_FORT": 0.4576, "train_start": "2000-11-15", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_mean_abs_ret_5d__prod__heston_xi", "Core_PCE_zscore_60d", "VIX_Price_zscore_60d__ret5x__MCD_ret_1d", "VIX_Price_zscore_60d__macross__AMD_zscore_60d", "ENB_EnbridgeInc_ret_5d__zrel__CMCSA_zscore_60d", "US7Y_Rate_ret_20d", "JNJ_ret_1d", "EQIX_Equinix_ret_5d", "TM_Telephone_ret_1d", "CVX_ret_20d__zrel__NFCI_ret_5d"], "is_new": false}, {"model_id": "v2_h3_STRESS_LightGBM_N11", "algo": "LightGBM", "regime": "STRESS", "horizon": 3, "n_features": 11, "F1_dir": 0.5447, "F1_UP_FORT": 0.2785, "F1_DOWN_FORT": 0.5, "train_start": "2000-11-15", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_mean_abs_ret_5d__prod__heston_xi", "Core_PCE_zscore_60d", "VIX_Price_zscore_60d__ret5x__MCD_ret_1d", "VIX_Price_zscore_60d__macross__AMD_zscore_60d", "ENB_EnbridgeInc_ret_5d__zrel__CMCSA_zscore_60d", "US7Y_Rate_ret_20d", "JNJ_ret_1d", "EQIX_Equinix_ret_5d", "TM_Telephone_ret_1d", "CVX_ret_20d__zrel__NFCI_ret_5d", "heston_xi__div__hmm_p_stress"], "is_new": false}, {"model_id": "v2_h3_STRESS_LightGBM_N12", "algo": "LightGBM", "regime": "STRESS", "horizon": 3, "n_features": 12, "F1_dir": 0.568, "F1_UP_FORT": 0.3214, "F1_DOWN_FORT": 0.4571, "train_start": "2000-11-15", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_mean_abs_ret_5d__prod__heston_xi", "Core_PCE_zscore_60d", "VIX_Price_zscore_60d__ret5x__MCD_ret_1d", "VIX_Price_zscore_60d__macross__AMD_zscore_60d", "ENB_EnbridgeInc_ret_5d__zrel__CMCSA_zscore_60d", "US7Y_Rate_ret_20d", "JNJ_ret_1d", "EQIX_Equinix_ret_5d", "TM_Telephone_ret_1d", "CVX_ret_20d__zrel__NFCI_ret_5d", "heston_xi__div__hmm_p_stress", "ENB_EnbridgeInc_ret_5d__div__hmm_p_stress"], "is_new": false}, {"model_id": "v2_h3_STRESS_LightGBM_N13", "algo": "LightGBM", "regime": "STRESS", "horizon": 3, "n_features": 13, "F1_dir": 0.5622, "F1_UP_FORT": 0.3145, "F1_DOWN_FORT": 0.4884, "train_start": "2000-11-15", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_mean_abs_ret_5d__prod__heston_xi", "Core_PCE_zscore_60d", "VIX_Price_zscore_60d__ret5x__MCD_ret_1d", "VIX_Price_zscore_60d__macross__AMD_zscore_60d", "ENB_EnbridgeInc_ret_5d__zrel__CMCSA_zscore_60d", "US7Y_Rate_ret_20d", "JNJ_ret_1d", "EQIX_Equinix_ret_5d", "TM_Telephone_ret_1d", "CVX_ret_20d__zrel__NFCI_ret_5d", "heston_xi__div__hmm_p_stress", "ENB_EnbridgeInc_ret_5d__div__hmm_p_stress", "COST_ret_5d__div__vix_vs_ma20"], "is_new": false}, {"model_id": "v2_h3_STRESS_LightGBM_N14", "algo": "LightGBM", "regime": "STRESS", "horizon": 3, "n_features": 14, "F1_dir": 0.5648, "F1_UP_FORT": 0.3354, "F1_DOWN_FORT": 0.4781, "train_start": "2000-11-15", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_mean_abs_ret_5d__prod__heston_xi", "Core_PCE_zscore_60d", "VIX_Price_zscore_60d__ret5x__MCD_ret_1d", "VIX_Price_zscore_60d__macross__AMD_zscore_60d", "ENB_EnbridgeInc_ret_5d__zrel__CMCSA_zscore_60d", "US7Y_Rate_ret_20d", "JNJ_ret_1d", "EQIX_Equinix_ret_5d", "TM_Telephone_ret_1d", "CVX_ret_20d__zrel__NFCI_ret_5d", "heston_xi__div__hmm_p_stress", "ENB_EnbridgeInc_ret_5d__div__hmm_p_stress", "COST_ret_5d__div__vix_vs_ma20", "hmm_p_stress"], "is_new": false}, {"model_id": "v2_h3_STRESS_LightGBM_N15", "algo": "LightGBM", "regime": "STRESS", "horizon": 3, "n_features": 15, "F1_dir": 0.5297, "F1_UP_FORT": 0.2345, "F1_DOWN_FORT": 0.4739, "train_start": "2000-11-15", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_mean_abs_ret_5d__prod__heston_xi", "Core_PCE_zscore_60d", "VIX_Price_zscore_60d__ret5x__MCD_ret_1d", "VIX_Price_zscore_60d__macross__AMD_zscore_60d", "ENB_EnbridgeInc_ret_5d__zrel__CMCSA_zscore_60d", "US7Y_Rate_ret_20d", "JNJ_ret_1d", "EQIX_Equinix_ret_5d", "TM_Telephone_ret_1d", "CVX_ret_20d__zrel__NFCI_ret_5d", "heston_xi__div__hmm_p_stress", "ENB_EnbridgeInc_ret_5d__div__hmm_p_stress", "COST_ret_5d__div__vix_vs_ma20", "hmm_p_stress", "PFE_ret_1d"], "is_new": false}, {"model_id": "v2_h3_STRESS_LightGBM_N16", "algo": "LightGBM", "regime": "STRESS", "horizon": 3, "n_features": 16, "F1_dir": 0.5406, "F1_UP_FORT": 0.2642, "F1_DOWN_FORT": 0.4643, "train_start": "2000-11-15", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_mean_abs_ret_5d__prod__heston_xi", "Core_PCE_zscore_60d", "VIX_Price_zscore_60d__ret5x__MCD_ret_1d", "VIX_Price_zscore_60d__macross__AMD_zscore_60d", "ENB_EnbridgeInc_ret_5d__zrel__CMCSA_zscore_60d", "US7Y_Rate_ret_20d", "JNJ_ret_1d", "EQIX_Equinix_ret_5d", "TM_Telephone_ret_1d", "CVX_ret_20d__zrel__NFCI_ret_5d", "heston_xi__div__hmm_p_stress", "ENB_EnbridgeInc_ret_5d__div__hmm_p_stress", "COST_ret_5d__div__vix_vs_ma20", "hmm_p_stress", "PFE_ret_1d", "ENB_EnbridgeInc_ret_5d__macross__CMCSA_zscore_60d"], "is_new": false}, {"model_id": "v2_h3_STRESS_LightGBM_N17", "algo": "LightGBM", "regime": "STRESS", "horizon": 3, "n_features": 17, "F1_dir": 0.5617, "F1_UP_FORT": 0.3394, "F1_DOWN_FORT": 0.471, "train_start": "2000-11-15", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_mean_abs_ret_5d__prod__heston_xi", "Core_PCE_zscore_60d", "VIX_Price_zscore_60d__ret5x__MCD_ret_1d", "VIX_Price_zscore_60d__macross__AMD_zscore_60d", "ENB_EnbridgeInc_ret_5d__zrel__CMCSA_zscore_60d", "US7Y_Rate_ret_20d", "JNJ_ret_1d", "EQIX_Equinix_ret_5d", "TM_Telephone_ret_1d", "CVX_ret_20d__zrel__NFCI_ret_5d", "heston_xi__div__hmm_p_stress", "ENB_EnbridgeInc_ret_5d__div__hmm_p_stress", "COST_ret_5d__div__vix_vs_ma20", "hmm_p_stress", "PFE_ret_1d", "ENB_EnbridgeInc_ret_5d__macross__CMCSA_zscore_60d", "VIX_Price_zscore_60d__ret5x__AMD_zscore_60d"], "is_new": false}, {"model_id": "v2_h3_STRESS_LightGBM_N18", "algo": "LightGBM", "regime": "STRESS", "horizon": 3, "n_features": 18, "F1_dir": 0.5144, "F1_UP_FORT": 0.2313, "F1_DOWN_FORT": 0.4605, "train_start": "2000-11-15", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_mean_abs_ret_5d__prod__heston_xi", "Core_PCE_zscore_60d", "VIX_Price_zscore_60d__ret5x__MCD_ret_1d", "VIX_Price_zscore_60d__macross__AMD_zscore_60d", "ENB_EnbridgeInc_ret_5d__zrel__CMCSA_zscore_60d", "US7Y_Rate_ret_20d", "JNJ_ret_1d", "EQIX_Equinix_ret_5d", "TM_Telephone_ret_1d", "CVX_ret_20d__zrel__NFCI_ret_5d", "heston_xi__div__hmm_p_stress", "ENB_EnbridgeInc_ret_5d__div__hmm_p_stress", "COST_ret_5d__div__vix_vs_ma20", "hmm_p_stress", "PFE_ret_1d", "ENB_EnbridgeInc_ret_5d__macross__CMCSA_zscore_60d", "VIX_Price_zscore_60d__ret5x__AMD_zscore_60d", "ORCL_zscore_60d"], "is_new": false}, {"model_id": "v2_h3_STRESS_GradientBoosting_N5", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 3, "n_features": 5, "F1_dir": 0.5224, "F1_UP_FORT": 0.3169, "F1_DOWN_FORT": 0.424, "train_start": "2000-11-15", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_mean_abs_ret_5d__prod__heston_xi", "Core_PCE_zscore_60d", "VIX_Price_zscore_60d__ret5x__MCD_ret_1d", "VIX_Price_zscore_60d__macross__AMD_zscore_60d", "ENB_EnbridgeInc_ret_5d__zrel__CMCSA_zscore_60d"], "is_new": false}, {"model_id": "v2_h3_STRESS_GradientBoosting_N6", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 3, "n_features": 6, "F1_dir": 0.5112, "F1_UP_FORT": 0.1757, "F1_DOWN_FORT": 0.4794, "train_start": "2000-11-15", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_mean_abs_ret_5d__prod__heston_xi", "Core_PCE_zscore_60d", "VIX_Price_zscore_60d__ret5x__MCD_ret_1d", "VIX_Price_zscore_60d__macross__AMD_zscore_60d", "ENB_EnbridgeInc_ret_5d__zrel__CMCSA_zscore_60d", "US7Y_Rate_ret_20d"], "is_new": false}, {"model_id": "v2_h3_STRESS_GradientBoosting_N7", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 3, "n_features": 7, "F1_dir": 0.509, "F1_UP_FORT": 0.1769, "F1_DOWN_FORT": 0.4565, "train_start": "2000-11-15", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_mean_abs_ret_5d__prod__heston_xi", "Core_PCE_zscore_60d", "VIX_Price_zscore_60d__ret5x__MCD_ret_1d", "VIX_Price_zscore_60d__macross__AMD_zscore_60d", "ENB_EnbridgeInc_ret_5d__zrel__CMCSA_zscore_60d", "US7Y_Rate_ret_20d", "JNJ_ret_1d"], "is_new": false}, {"model_id": "v2_h3_STRESS_GradientBoosting_N10", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 3, "n_features": 10, "F1_dir": 0.5387, "F1_UP_FORT": 0.2338, "F1_DOWN_FORT": 0.4355, "train_start": "2000-11-15", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_mean_abs_ret_5d__prod__heston_xi", "Core_PCE_zscore_60d", "VIX_Price_zscore_60d__ret5x__MCD_ret_1d", "VIX_Price_zscore_60d__macross__AMD_zscore_60d", "ENB_EnbridgeInc_ret_5d__zrel__CMCSA_zscore_60d", "US7Y_Rate_ret_20d", "JNJ_ret_1d", "EQIX_Equinix_ret_5d", "TM_Telephone_ret_1d", "CVX_ret_20d__zrel__NFCI_ret_5d"], "is_new": false}, {"model_id": "v2_h3_STRESS_GradientBoosting_N11", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 3, "n_features": 11, "F1_dir": 0.5517, "F1_UP_FORT": 0.2649, "F1_DOWN_FORT": 0.4604, "train_start": "2000-11-15", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_mean_abs_ret_5d__prod__heston_xi", "Core_PCE_zscore_60d", "VIX_Price_zscore_60d__ret5x__MCD_ret_1d", "VIX_Price_zscore_60d__macross__AMD_zscore_60d", "ENB_EnbridgeInc_ret_5d__zrel__CMCSA_zscore_60d", "US7Y_Rate_ret_20d", "JNJ_ret_1d", "EQIX_Equinix_ret_5d", "TM_Telephone_ret_1d", "CVX_ret_20d__zrel__NFCI_ret_5d", "heston_xi__div__hmm_p_stress"], "is_new": false}, {"model_id": "v2_h3_STRESS_GradientBoosting_N12", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 3, "n_features": 12, "F1_dir": 0.5493, "F1_UP_FORT": 0.2981, "F1_DOWN_FORT": 0.4519, "train_start": "2000-11-15", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_mean_abs_ret_5d__prod__heston_xi", "Core_PCE_zscore_60d", "VIX_Price_zscore_60d__ret5x__MCD_ret_1d", "VIX_Price_zscore_60d__macross__AMD_zscore_60d", "ENB_EnbridgeInc_ret_5d__zrel__CMCSA_zscore_60d", "US7Y_Rate_ret_20d", "JNJ_ret_1d", "EQIX_Equinix_ret_5d", "TM_Telephone_ret_1d", "CVX_ret_20d__zrel__NFCI_ret_5d", "heston_xi__div__hmm_p_stress", "ENB_EnbridgeInc_ret_5d__div__hmm_p_stress"], "is_new": false}, {"model_id": "v2_h3_STRESS_GradientBoosting_N13", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 3, "n_features": 13, "F1_dir": 0.5302, "F1_UP_FORT": 0.2517, "F1_DOWN_FORT": 0.4487, "train_start": "2000-11-15", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_mean_abs_ret_5d__prod__heston_xi", "Core_PCE_zscore_60d", "VIX_Price_zscore_60d__ret5x__MCD_ret_1d", "VIX_Price_zscore_60d__macross__AMD_zscore_60d", "ENB_EnbridgeInc_ret_5d__zrel__CMCSA_zscore_60d", "US7Y_Rate_ret_20d", "JNJ_ret_1d", "EQIX_Equinix_ret_5d", "TM_Telephone_ret_1d", "CVX_ret_20d__zrel__NFCI_ret_5d", "heston_xi__div__hmm_p_stress", "ENB_EnbridgeInc_ret_5d__div__hmm_p_stress", "COST_ret_5d__div__vix_vs_ma20"], "is_new": false}, {"model_id": "v2_h3_STRESS_GradientBoosting_N14", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 3, "n_features": 14, "F1_dir": 0.5443, "F1_UP_FORT": 0.2963, "F1_DOWN_FORT": 0.4521, "train_start": "2000-11-15", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_mean_abs_ret_5d__prod__heston_xi", "Core_PCE_zscore_60d", "VIX_Price_zscore_60d__ret5x__MCD_ret_1d", "VIX_Price_zscore_60d__macross__AMD_zscore_60d", "ENB_EnbridgeInc_ret_5d__zrel__CMCSA_zscore_60d", "US7Y_Rate_ret_20d", "JNJ_ret_1d", "EQIX_Equinix_ret_5d", "TM_Telephone_ret_1d", "CVX_ret_20d__zrel__NFCI_ret_5d", "heston_xi__div__hmm_p_stress", "ENB_EnbridgeInc_ret_5d__div__hmm_p_stress", "COST_ret_5d__div__vix_vs_ma20", "hmm_p_stress"], "is_new": false}, {"model_id": "v2_h3_STRESS_GradientBoosting_N15", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 3, "n_features": 15, "F1_dir": 0.5408, "F1_UP_FORT": 0.2548, "F1_DOWN_FORT": 0.4532, "train_start": "2000-11-15", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_mean_abs_ret_5d__prod__heston_xi", "Core_PCE_zscore_60d", "VIX_Price_zscore_60d__ret5x__MCD_ret_1d", "VIX_Price_zscore_60d__macross__AMD_zscore_60d", "ENB_EnbridgeInc_ret_5d__zrel__CMCSA_zscore_60d", "US7Y_Rate_ret_20d", "JNJ_ret_1d", "EQIX_Equinix_ret_5d", "TM_Telephone_ret_1d", "CVX_ret_20d__zrel__NFCI_ret_5d", "heston_xi__div__hmm_p_stress", "ENB_EnbridgeInc_ret_5d__div__hmm_p_stress", "COST_ret_5d__div__vix_vs_ma20", "hmm_p_stress", "PFE_ret_1d"], "is_new": false}, {"model_id": "v2_h3_STRESS_GradientBoosting_N16", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 3, "n_features": 16, "F1_dir": 0.5236, "F1_UP_FORT": 0.2466, "F1_DOWN_FORT": 0.4484, "train_start": "2000-11-15", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_mean_abs_ret_5d__prod__heston_xi", "Core_PCE_zscore_60d", "VIX_Price_zscore_60d__ret5x__MCD_ret_1d", "VIX_Price_zscore_60d__macross__AMD_zscore_60d", "ENB_EnbridgeInc_ret_5d__zrel__CMCSA_zscore_60d", "US7Y_Rate_ret_20d", "JNJ_ret_1d", "EQIX_Equinix_ret_5d", "TM_Telephone_ret_1d", "CVX_ret_20d__zrel__NFCI_ret_5d", "heston_xi__div__hmm_p_stress", "ENB_EnbridgeInc_ret_5d__div__hmm_p_stress", "COST_ret_5d__div__vix_vs_ma20", "hmm_p_stress", "PFE_ret_1d", "ENB_EnbridgeInc_ret_5d__macross__CMCSA_zscore_60d"], "is_new": false}, {"model_id": "v2_h3_STRESS_GradientBoosting_N17", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 3, "n_features": 17, "F1_dir": 0.5281, "F1_UP_FORT": 0.2468, "F1_DOWN_FORT": 0.4514, "train_start": "2000-11-15", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_mean_abs_ret_5d__prod__heston_xi", "Core_PCE_zscore_60d", "VIX_Price_zscore_60d__ret5x__MCD_ret_1d", "VIX_Price_zscore_60d__macross__AMD_zscore_60d", "ENB_EnbridgeInc_ret_5d__zrel__CMCSA_zscore_60d", "US7Y_Rate_ret_20d", "JNJ_ret_1d", "EQIX_Equinix_ret_5d", "TM_Telephone_ret_1d", "CVX_ret_20d__zrel__NFCI_ret_5d", "heston_xi__div__hmm_p_stress", "ENB_EnbridgeInc_ret_5d__div__hmm_p_stress", "COST_ret_5d__div__vix_vs_ma20", "hmm_p_stress", "PFE_ret_1d", "ENB_EnbridgeInc_ret_5d__macross__CMCSA_zscore_60d", "VIX_Price_zscore_60d__ret5x__AMD_zscore_60d"], "is_new": false}, {"model_id": "v2_h3_STRESS_GradientBoosting_N18", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 3, "n_features": 18, "F1_dir": 0.5089, "F1_UP_FORT": 0.2297, "F1_DOWN_FORT": 0.4702, "train_start": "2000-11-15", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_mean_abs_ret_5d__prod__heston_xi", "Core_PCE_zscore_60d", "VIX_Price_zscore_60d__ret5x__MCD_ret_1d", "VIX_Price_zscore_60d__macross__AMD_zscore_60d", "ENB_EnbridgeInc_ret_5d__zrel__CMCSA_zscore_60d", "US7Y_Rate_ret_20d", "JNJ_ret_1d", "EQIX_Equinix_ret_5d", "TM_Telephone_ret_1d", "CVX_ret_20d__zrel__NFCI_ret_5d", "heston_xi__div__hmm_p_stress", "ENB_EnbridgeInc_ret_5d__div__hmm_p_stress", "COST_ret_5d__div__vix_vs_ma20", "hmm_p_stress", "PFE_ret_1d", "ENB_EnbridgeInc_ret_5d__macross__CMCSA_zscore_60d", "VIX_Price_zscore_60d__ret5x__AMD_zscore_60d", "ORCL_zscore_60d"], "is_new": false}, {"model_id": "v2_h3_STRESS_RandomForest_N5", "algo": "RandomForest", "regime": "STRESS", "horizon": 3, "n_features": 5, "F1_dir": 0.529, "F1_UP_FORT": 0.2857, "F1_DOWN_FORT": 0.4706, "train_start": "2000-11-15", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_mean_abs_ret_5d__prod__heston_xi", "Core_PCE_zscore_60d", "VIX_Price_zscore_60d__ret5x__MCD_ret_1d", "VIX_Price_zscore_60d__macross__AMD_zscore_60d", "ENB_EnbridgeInc_ret_5d__zrel__CMCSA_zscore_60d"], "is_new": false}, {"model_id": "v2_h3_STRESS_RandomForest_N7", "algo": "RandomForest", "regime": "STRESS", "horizon": 3, "n_features": 7, "F1_dir": 0.5094, "F1_UP_FORT": 0.1167, "F1_DOWN_FORT": 0.4969, "train_start": "2000-11-15", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_mean_abs_ret_5d__prod__heston_xi", "Core_PCE_zscore_60d", "VIX_Price_zscore_60d__ret5x__MCD_ret_1d", "VIX_Price_zscore_60d__macross__AMD_zscore_60d", "ENB_EnbridgeInc_ret_5d__zrel__CMCSA_zscore_60d", "US7Y_Rate_ret_20d", "JNJ_ret_1d"], "is_new": false}, {"model_id": "v2_h3_STRESS_RandomForest_N10", "algo": "RandomForest", "regime": "STRESS", "horizon": 3, "n_features": 10, "F1_dir": 0.5485, "F1_UP_FORT": 0.3537, "F1_DOWN_FORT": 0.4735, "train_start": "2000-11-15", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_mean_abs_ret_5d__prod__heston_xi", "Core_PCE_zscore_60d", "VIX_Price_zscore_60d__ret5x__MCD_ret_1d", "VIX_Price_zscore_60d__macross__AMD_zscore_60d", "ENB_EnbridgeInc_ret_5d__zrel__CMCSA_zscore_60d", "US7Y_Rate_ret_20d", "JNJ_ret_1d", "EQIX_Equinix_ret_5d", "TM_Telephone_ret_1d", "CVX_ret_20d__zrel__NFCI_ret_5d"], "is_new": false}, {"model_id": "v2_h3_STRESS_RandomForest_N11", "algo": "RandomForest", "regime": "STRESS", "horizon": 3, "n_features": 11, "F1_dir": 0.5552, "F1_UP_FORT": 0.3667, "F1_DOWN_FORT": 0.5205, "train_start": "2000-11-15", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_mean_abs_ret_5d__prod__heston_xi", "Core_PCE_zscore_60d", "VIX_Price_zscore_60d__ret5x__MCD_ret_1d", "VIX_Price_zscore_60d__macross__AMD_zscore_60d", "ENB_EnbridgeInc_ret_5d__zrel__CMCSA_zscore_60d", "US7Y_Rate_ret_20d", "JNJ_ret_1d", "EQIX_Equinix_ret_5d", "TM_Telephone_ret_1d", "CVX_ret_20d__zrel__NFCI_ret_5d", "heston_xi__div__hmm_p_stress"], "is_new": false}, {"model_id": "v2_h3_STRESS_RandomForest_N12", "algo": "RandomForest", "regime": "STRESS", "horizon": 3, "n_features": 12, "F1_dir": 0.5359, "F1_UP_FORT": 0.3234, "F1_DOWN_FORT": 0.5051, "train_start": "2000-11-15", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_mean_abs_ret_5d__prod__heston_xi", "Core_PCE_zscore_60d", "VIX_Price_zscore_60d__ret5x__MCD_ret_1d", "VIX_Price_zscore_60d__macross__AMD_zscore_60d", "ENB_EnbridgeInc_ret_5d__zrel__CMCSA_zscore_60d", "US7Y_Rate_ret_20d", "JNJ_ret_1d", "EQIX_Equinix_ret_5d", "TM_Telephone_ret_1d", "CVX_ret_20d__zrel__NFCI_ret_5d", "heston_xi__div__hmm_p_stress", "ENB_EnbridgeInc_ret_5d__div__hmm_p_stress"], "is_new": false}, {"model_id": "v2_h3_STRESS_RandomForest_N13", "algo": "RandomForest", "regime": "STRESS", "horizon": 3, "n_features": 13, "F1_dir": 0.5635, "F1_UP_FORT": 0.3333, "F1_DOWN_FORT": 0.52, "train_start": "2000-11-15", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_mean_abs_ret_5d__prod__heston_xi", "Core_PCE_zscore_60d", "VIX_Price_zscore_60d__ret5x__MCD_ret_1d", "VIX_Price_zscore_60d__macross__AMD_zscore_60d", "ENB_EnbridgeInc_ret_5d__zrel__CMCSA_zscore_60d", "US7Y_Rate_ret_20d", "JNJ_ret_1d", "EQIX_Equinix_ret_5d", "TM_Telephone_ret_1d", "CVX_ret_20d__zrel__NFCI_ret_5d", "heston_xi__div__hmm_p_stress", "ENB_EnbridgeInc_ret_5d__div__hmm_p_stress", "COST_ret_5d__div__vix_vs_ma20"], "is_new": false}, {"model_id": "v2_h3_STRESS_RandomForest_N14", "algo": "RandomForest", "regime": "STRESS", "horizon": 3, "n_features": 14, "F1_dir": 0.5564, "F1_UP_FORT": 0.3832, "F1_DOWN_FORT": 0.5288, "train_start": "2000-11-15", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_mean_abs_ret_5d__prod__heston_xi", "Core_PCE_zscore_60d", "VIX_Price_zscore_60d__ret5x__MCD_ret_1d", "VIX_Price_zscore_60d__macross__AMD_zscore_60d", "ENB_EnbridgeInc_ret_5d__zrel__CMCSA_zscore_60d", "US7Y_Rate_ret_20d", "JNJ_ret_1d", "EQIX_Equinix_ret_5d", "TM_Telephone_ret_1d", "CVX_ret_20d__zrel__NFCI_ret_5d", "heston_xi__div__hmm_p_stress", "ENB_EnbridgeInc_ret_5d__div__hmm_p_stress", "COST_ret_5d__div__vix_vs_ma20", "hmm_p_stress"], "is_new": false}, {"model_id": "v2_h3_STRESS_RandomForest_N15", "algo": "RandomForest", "regime": "STRESS", "horizon": 3, "n_features": 15, "F1_dir": 0.5721, "F1_UP_FORT": 0.3657, "F1_DOWN_FORT": 0.5306, "train_start": "2000-11-15", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_mean_abs_ret_5d__prod__heston_xi", "Core_PCE_zscore_60d", "VIX_Price_zscore_60d__ret5x__MCD_ret_1d", "VIX_Price_zscore_60d__macross__AMD_zscore_60d", "ENB_EnbridgeInc_ret_5d__zrel__CMCSA_zscore_60d", "US7Y_Rate_ret_20d", "JNJ_ret_1d", "EQIX_Equinix_ret_5d", "TM_Telephone_ret_1d", "CVX_ret_20d__zrel__NFCI_ret_5d", "heston_xi__div__hmm_p_stress", "ENB_EnbridgeInc_ret_5d__div__hmm_p_stress", "COST_ret_5d__div__vix_vs_ma20", "hmm_p_stress", "PFE_ret_1d"], "is_new": false}, {"model_id": "v2_h3_STRESS_RandomForest_N16", "algo": "RandomForest", "regime": "STRESS", "horizon": 3, "n_features": 16, "F1_dir": 0.5516, "F1_UP_FORT": 0.3793, "F1_DOWN_FORT": 0.5133, "train_start": "2000-11-15", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_mean_abs_ret_5d__prod__heston_xi", "Core_PCE_zscore_60d", "VIX_Price_zscore_60d__ret5x__MCD_ret_1d", "VIX_Price_zscore_60d__macross__AMD_zscore_60d", "ENB_EnbridgeInc_ret_5d__zrel__CMCSA_zscore_60d", "US7Y_Rate_ret_20d", "JNJ_ret_1d", "EQIX_Equinix_ret_5d", "TM_Telephone_ret_1d", "CVX_ret_20d__zrel__NFCI_ret_5d", "heston_xi__div__hmm_p_stress", "ENB_EnbridgeInc_ret_5d__div__hmm_p_stress", "COST_ret_5d__div__vix_vs_ma20", "hmm_p_stress", "PFE_ret_1d", "ENB_EnbridgeInc_ret_5d__macross__CMCSA_zscore_60d"], "is_new": false}, {"model_id": "v2_h3_STRESS_RandomForest_N17", "algo": "RandomForest", "regime": "STRESS", "horizon": 3, "n_features": 17, "F1_dir": 0.543, "F1_UP_FORT": 0.3832, "F1_DOWN_FORT": 0.5084, "train_start": "2000-11-15", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_mean_abs_ret_5d__prod__heston_xi", "Core_PCE_zscore_60d", "VIX_Price_zscore_60d__ret5x__MCD_ret_1d", "VIX_Price_zscore_60d__macross__AMD_zscore_60d", "ENB_EnbridgeInc_ret_5d__zrel__CMCSA_zscore_60d", "US7Y_Rate_ret_20d", "JNJ_ret_1d", "EQIX_Equinix_ret_5d", "TM_Telephone_ret_1d", "CVX_ret_20d__zrel__NFCI_ret_5d", "heston_xi__div__hmm_p_stress", "ENB_EnbridgeInc_ret_5d__div__hmm_p_stress", "COST_ret_5d__div__vix_vs_ma20", "hmm_p_stress", "PFE_ret_1d", "ENB_EnbridgeInc_ret_5d__macross__CMCSA_zscore_60d", "VIX_Price_zscore_60d__ret5x__AMD_zscore_60d"], "is_new": false}, {"model_id": "v2_h3_STRESS_RandomForest_N18", "algo": "RandomForest", "regime": "STRESS", "horizon": 3, "n_features": 18, "F1_dir": 0.5406, "F1_UP_FORT": 0.3137, "F1_DOWN_FORT": 0.5223, "train_start": "2000-11-15", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_mean_abs_ret_5d__prod__heston_xi", "Core_PCE_zscore_60d", "VIX_Price_zscore_60d__ret5x__MCD_ret_1d", "VIX_Price_zscore_60d__macross__AMD_zscore_60d", "ENB_EnbridgeInc_ret_5d__zrel__CMCSA_zscore_60d", "US7Y_Rate_ret_20d", "JNJ_ret_1d", "EQIX_Equinix_ret_5d", "TM_Telephone_ret_1d", "CVX_ret_20d__zrel__NFCI_ret_5d", "heston_xi__div__hmm_p_stress", "ENB_EnbridgeInc_ret_5d__div__hmm_p_stress", "COST_ret_5d__div__vix_vs_ma20", "hmm_p_stress", "PFE_ret_1d", "ENB_EnbridgeInc_ret_5d__macross__CMCSA_zscore_60d", "VIX_Price_zscore_60d__ret5x__AMD_zscore_60d", "ORCL_zscore_60d"], "is_new": false}, {"model_id": "v2_h3_STRESS_LogisticRegression_N10", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 3, "n_features": 10, "F1_dir": 0.5234, "F1_UP_FORT": 0.3708, "F1_DOWN_FORT": 0.493, "train_start": "2000-11-15", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_mean_abs_ret_5d__prod__heston_xi", "Core_PCE_zscore_60d", "VIX_Price_zscore_60d__ret5x__MCD_ret_1d", "VIX_Price_zscore_60d__macross__AMD_zscore_60d", "ENB_EnbridgeInc_ret_5d__zrel__CMCSA_zscore_60d", "US7Y_Rate_ret_20d", "JNJ_ret_1d", "EQIX_Equinix_ret_5d", "TM_Telephone_ret_1d", "CVX_ret_20d__zrel__NFCI_ret_5d"], "is_new": false}, {"model_id": "v2_h3_STRESS_LogisticRegression_N11", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 3, "n_features": 11, "F1_dir": 0.5425, "F1_UP_FORT": 0.3696, "F1_DOWN_FORT": 0.4875, "train_start": "2000-11-15", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_mean_abs_ret_5d__prod__heston_xi", "Core_PCE_zscore_60d", "VIX_Price_zscore_60d__ret5x__MCD_ret_1d", "VIX_Price_zscore_60d__macross__AMD_zscore_60d", "ENB_EnbridgeInc_ret_5d__zrel__CMCSA_zscore_60d", "US7Y_Rate_ret_20d", "JNJ_ret_1d", "EQIX_Equinix_ret_5d", "TM_Telephone_ret_1d", "CVX_ret_20d__zrel__NFCI_ret_5d", "heston_xi__div__hmm_p_stress"], "is_new": false}, {"model_id": "v2_h3_STRESS_LogisticRegression_N12", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 3, "n_features": 12, "F1_dir": 0.5437, "F1_UP_FORT": 0.3607, "F1_DOWN_FORT": 0.4875, "train_start": "2000-11-15", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_mean_abs_ret_5d__prod__heston_xi", "Core_PCE_zscore_60d", "VIX_Price_zscore_60d__ret5x__MCD_ret_1d", "VIX_Price_zscore_60d__macross__AMD_zscore_60d", "ENB_EnbridgeInc_ret_5d__zrel__CMCSA_zscore_60d", "US7Y_Rate_ret_20d", "JNJ_ret_1d", "EQIX_Equinix_ret_5d", "TM_Telephone_ret_1d", "CVX_ret_20d__zrel__NFCI_ret_5d", "heston_xi__div__hmm_p_stress", "ENB_EnbridgeInc_ret_5d__div__hmm_p_stress"], "is_new": false}, {"model_id": "v2_h3_STRESS_LogisticRegression_N13", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 3, "n_features": 13, "F1_dir": 0.5484, "F1_UP_FORT": 0.3626, "F1_DOWN_FORT": 0.4946, "train_start": "2000-11-15", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_mean_abs_ret_5d__prod__heston_xi", "Core_PCE_zscore_60d", "VIX_Price_zscore_60d__ret5x__MCD_ret_1d", "VIX_Price_zscore_60d__macross__AMD_zscore_60d", "ENB_EnbridgeInc_ret_5d__zrel__CMCSA_zscore_60d", "US7Y_Rate_ret_20d", "JNJ_ret_1d", "EQIX_Equinix_ret_5d", "TM_Telephone_ret_1d", "CVX_ret_20d__zrel__NFCI_ret_5d", "heston_xi__div__hmm_p_stress", "ENB_EnbridgeInc_ret_5d__div__hmm_p_stress", "COST_ret_5d__div__vix_vs_ma20"], "is_new": false}, {"model_id": "v2_h3_STRESS_LogisticRegression_N14", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 3, "n_features": 14, "F1_dir": 0.5483, "F1_UP_FORT": 0.3804, "F1_DOWN_FORT": 0.4895, "train_start": "2000-11-15", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_mean_abs_ret_5d__prod__heston_xi", "Core_PCE_zscore_60d", "VIX_Price_zscore_60d__ret5x__MCD_ret_1d", "VIX_Price_zscore_60d__macross__AMD_zscore_60d", "ENB_EnbridgeInc_ret_5d__zrel__CMCSA_zscore_60d", "US7Y_Rate_ret_20d", "JNJ_ret_1d", "EQIX_Equinix_ret_5d", "TM_Telephone_ret_1d", "CVX_ret_20d__zrel__NFCI_ret_5d", "heston_xi__div__hmm_p_stress", "ENB_EnbridgeInc_ret_5d__div__hmm_p_stress", "COST_ret_5d__div__vix_vs_ma20", "hmm_p_stress"], "is_new": false}, {"model_id": "v2_h3_STRESS_LogisticRegression_N15", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 3, "n_features": 15, "F1_dir": 0.5519, "F1_UP_FORT": 0.3892, "F1_DOWN_FORT": 0.4876, "train_start": "2000-11-15", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_mean_abs_ret_5d__prod__heston_xi", "Core_PCE_zscore_60d", "VIX_Price_zscore_60d__ret5x__MCD_ret_1d", "VIX_Price_zscore_60d__macross__AMD_zscore_60d", "ENB_EnbridgeInc_ret_5d__zrel__CMCSA_zscore_60d", "US7Y_Rate_ret_20d", "JNJ_ret_1d", "EQIX_Equinix_ret_5d", "TM_Telephone_ret_1d", "CVX_ret_20d__zrel__NFCI_ret_5d", "heston_xi__div__hmm_p_stress", "ENB_EnbridgeInc_ret_5d__div__hmm_p_stress", "COST_ret_5d__div__vix_vs_ma20", "hmm_p_stress", "PFE_ret_1d"], "is_new": false}, {"model_id": "v2_h3_STRESS_LogisticRegression_N16", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 3, "n_features": 16, "F1_dir": 0.5472, "F1_UP_FORT": 0.3978, "F1_DOWN_FORT": 0.4823, "train_start": "2000-11-15", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_mean_abs_ret_5d__prod__heston_xi", "Core_PCE_zscore_60d", "VIX_Price_zscore_60d__ret5x__MCD_ret_1d", "VIX_Price_zscore_60d__macross__AMD_zscore_60d", "ENB_EnbridgeInc_ret_5d__zrel__CMCSA_zscore_60d", "US7Y_Rate_ret_20d", "JNJ_ret_1d", "EQIX_Equinix_ret_5d", "TM_Telephone_ret_1d", "CVX_ret_20d__zrel__NFCI_ret_5d", "heston_xi__div__hmm_p_stress", "ENB_EnbridgeInc_ret_5d__div__hmm_p_stress", "COST_ret_5d__div__vix_vs_ma20", "hmm_p_stress", "PFE_ret_1d", "ENB_EnbridgeInc_ret_5d__macross__CMCSA_zscore_60d"], "is_new": false}, {"model_id": "v2_h3_STRESS_LogisticRegression_N17", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 3, "n_features": 17, "F1_dir": 0.5437, "F1_UP_FORT": 0.3936, "F1_DOWN_FORT": 0.4823, "train_start": "2000-11-15", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_mean_abs_ret_5d__prod__heston_xi", "Core_PCE_zscore_60d", "VIX_Price_zscore_60d__ret5x__MCD_ret_1d", "VIX_Price_zscore_60d__macross__AMD_zscore_60d", "ENB_EnbridgeInc_ret_5d__zrel__CMCSA_zscore_60d", "US7Y_Rate_ret_20d", "JNJ_ret_1d", "EQIX_Equinix_ret_5d", "TM_Telephone_ret_1d", "CVX_ret_20d__zrel__NFCI_ret_5d", "heston_xi__div__hmm_p_stress", "ENB_EnbridgeInc_ret_5d__div__hmm_p_stress", "COST_ret_5d__div__vix_vs_ma20", "hmm_p_stress", "PFE_ret_1d", "ENB_EnbridgeInc_ret_5d__macross__CMCSA_zscore_60d", "VIX_Price_zscore_60d__ret5x__AMD_zscore_60d"], "is_new": false}, {"model_id": "v2_h3_STRESS_LogisticRegression_N18", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 3, "n_features": 18, "F1_dir": 0.519, "F1_UP_FORT": 0.3656, "F1_DOWN_FORT": 0.4638, "train_start": "2000-11-15", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_mean_abs_ret_5d__prod__heston_xi", "Core_PCE_zscore_60d", "VIX_Price_zscore_60d__ret5x__MCD_ret_1d", "VIX_Price_zscore_60d__macross__AMD_zscore_60d", "ENB_EnbridgeInc_ret_5d__zrel__CMCSA_zscore_60d", "US7Y_Rate_ret_20d", "JNJ_ret_1d", "EQIX_Equinix_ret_5d", "TM_Telephone_ret_1d", "CVX_ret_20d__zrel__NFCI_ret_5d", "heston_xi__div__hmm_p_stress", "ENB_EnbridgeInc_ret_5d__div__hmm_p_stress", "COST_ret_5d__div__vix_vs_ma20", "hmm_p_stress", "PFE_ret_1d", "ENB_EnbridgeInc_ret_5d__macross__CMCSA_zscore_60d", "VIX_Price_zscore_60d__ret5x__AMD_zscore_60d", "ORCL_zscore_60d"], "is_new": false}, {"model_id": "v2_h3_STRESS_RandomForest_Optuna_N14", "algo": "RandomForest", "regime": "STRESS", "horizon": 3, "n_features": 14, "F1_dir": 0.5539, "F1_UP_FORT": 0.3598, "F1_DOWN_FORT": 0.5106, "train_start": "2000-11-15", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_mean_abs_ret_5d__prod__heston_xi", "Core_PCE_zscore_60d", "VIX_Price_zscore_60d__ret5x__MCD_ret_1d", "VIX_Price_zscore_60d__macross__AMD_zscore_60d", "ENB_EnbridgeInc_ret_5d__zrel__CMCSA_zscore_60d", "US7Y_Rate_ret_20d", "JNJ_ret_1d", "EQIX_Equinix_ret_5d", "TM_Telephone_ret_1d", "CVX_ret_20d__zrel__NFCI_ret_5d", "heston_xi__div__hmm_p_stress", "ENB_EnbridgeInc_ret_5d__div__hmm_p_stress", "COST_ret_5d__div__vix_vs_ma20", "hmm_p_stress"], "is_new": false}, {"model_id": "v2_h3_STRESS_RandomForest_OptunaCal_N14", "algo": "RandomForestCal", "regime": "STRESS", "horizon": 3, "n_features": 14, "F1_dir": 0.5497, "F1_UP_FORT": 0.3404, "F1_DOWN_FORT": 0.4981, "train_start": "2000-11-15", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_mean_abs_ret_5d__prod__heston_xi", "Core_PCE_zscore_60d", "VIX_Price_zscore_60d__ret5x__MCD_ret_1d", "VIX_Price_zscore_60d__macross__AMD_zscore_60d", "ENB_EnbridgeInc_ret_5d__zrel__CMCSA_zscore_60d", "US7Y_Rate_ret_20d", "JNJ_ret_1d", "EQIX_Equinix_ret_5d", "TM_Telephone_ret_1d", "CVX_ret_20d__zrel__NFCI_ret_5d", "heston_xi__div__hmm_p_stress", "ENB_EnbridgeInc_ret_5d__div__hmm_p_stress", "COST_ret_5d__div__vix_vs_ma20", "hmm_p_stress"], "is_new": false}, {"model_id": "v2_h3_GLOBAL_XGBoost_N5", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 3, "n_features": 5, "F1_dir": 0.6197, "F1_UP_FORT": 0.3818, "F1_DOWN_FORT": 0.4664, "train_start": "2007-01-31", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["kalman_innovation__div__kalman_residual", "kalman_innovation__macross__kalman_residual", "PAYX_Paychex_vol_20d", "VIX_Price_zscore_60d__div__vix_max_abs_ret_5d", "TXN_vol_20d__div__heston_xi"], "is_new": false}, {"model_id": "v2_h3_GLOBAL_LightGBM_N5", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 3, "n_features": 5, "F1_dir": 0.6015, "F1_UP_FORT": 0.362, "F1_DOWN_FORT": 0.4407, "train_start": "2007-01-31", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["kalman_innovation__div__kalman_residual", "kalman_innovation__macross__kalman_residual", "PAYX_Paychex_vol_20d", "VIX_Price_zscore_60d__div__vix_max_abs_ret_5d", "TXN_vol_20d__div__heston_xi"], "is_new": false}, {"model_id": "v2_h3_GLOBAL_LightGBM_N12", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 3, "n_features": 12, "F1_dir": 0.6789, "F1_UP_FORT": 0.5024, "F1_DOWN_FORT": 0.5444, "train_start": "2007-01-31", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["kalman_innovation__div__kalman_residual", "kalman_innovation__macross__kalman_residual", "PAYX_Paychex_vol_20d", "VIX_Price_zscore_60d__div__vix_max_abs_ret_5d", "TXN_vol_20d__div__heston_xi", "TXN_vol_20d__div__kalman_residual", "CCI_CrownCastle_vol_20d", "VIX_Price_zscore_60d__minus__NFCI_zscore_60d", "vix_zscore_10d__minus__EWQ_France_zscore_60d", "AMZN_ret_5d", "ITT_ITTInc_ret_5d", "VIX_Price_zscore_60d__div__hmm_p_stress"], "is_new": false}, {"model_id": "v2_h3_GLOBAL_LightGBM_N14", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 3, "n_features": 14, "F1_dir": 0.6796, "F1_UP_FORT": 0.5245, "F1_DOWN_FORT": 0.5528, "train_start": "2007-01-31", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["kalman_innovation__div__kalman_residual", "kalman_innovation__macross__kalman_residual", "PAYX_Paychex_vol_20d", "VIX_Price_zscore_60d__div__vix_max_abs_ret_5d", "TXN_vol_20d__div__heston_xi", "TXN_vol_20d__div__kalman_residual", "CCI_CrownCastle_vol_20d", "VIX_Price_zscore_60d__minus__NFCI_zscore_60d", "vix_zscore_10d__minus__EWQ_France_zscore_60d", "AMZN_ret_5d", "ITT_ITTInc_ret_5d", "VIX_Price_zscore_60d__div__hmm_p_stress", "EWL_Switzerland_vol_20d", "VVIX_ret_20d"], "is_new": false}, {"model_id": "v2_h3_GLOBAL_LightGBM_N16", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 3, "n_features": 16, "F1_dir": 0.6765, "F1_UP_FORT": 0.5325, "F1_DOWN_FORT": 0.5549, "train_start": "2007-01-31", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["kalman_innovation__div__kalman_residual", "kalman_innovation__macross__kalman_residual", "PAYX_Paychex_vol_20d", "VIX_Price_zscore_60d__div__vix_max_abs_ret_5d", "TXN_vol_20d__div__heston_xi", "TXN_vol_20d__div__kalman_residual", "CCI_CrownCastle_vol_20d", "VIX_Price_zscore_60d__minus__NFCI_zscore_60d", "vix_zscore_10d__minus__EWQ_France_zscore_60d", "AMZN_ret_5d", "ITT_ITTInc_ret_5d", "VIX_Price_zscore_60d__div__hmm_p_stress", "EWL_Switzerland_vol_20d", "VVIX_ret_20d", "vix_vol_of_vol_10d__div__kalman_residual", "heston_xi__div__kalman_residual"], "is_new": false}, {"model_id": "v2_h3_GLOBAL_GradientBoosting_N5", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 3, "n_features": 5, "F1_dir": 0.6124, "F1_UP_FORT": 0.3846, "F1_DOWN_FORT": 0.4643, "train_start": "2007-01-31", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["kalman_innovation__div__kalman_residual", "kalman_innovation__macross__kalman_residual", "PAYX_Paychex_vol_20d", "VIX_Price_zscore_60d__div__vix_max_abs_ret_5d", "TXN_vol_20d__div__heston_xi"], "is_new": false}, {"model_id": "v2_h3_GLOBAL_RandomForest_N5", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 3, "n_features": 5, "F1_dir": 0.6169, "F1_UP_FORT": 0.3962, "F1_DOWN_FORT": 0.4656, "train_start": "2007-01-31", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["kalman_innovation__div__kalman_residual", "kalman_innovation__macross__kalman_residual", "PAYX_Paychex_vol_20d", "VIX_Price_zscore_60d__div__vix_max_abs_ret_5d", "TXN_vol_20d__div__heston_xi"], "is_new": false}, {"model_id": "v2_h3_GLOBAL_RandomForest_N13", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 3, "n_features": 13, "F1_dir": 0.675, "F1_UP_FORT": 0.4615, "F1_DOWN_FORT": 0.5248, "train_start": "2007-01-31", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["kalman_innovation__div__kalman_residual", "kalman_innovation__macross__kalman_residual", "PAYX_Paychex_vol_20d", "VIX_Price_zscore_60d__div__vix_max_abs_ret_5d", "TXN_vol_20d__div__heston_xi", "TXN_vol_20d__div__kalman_residual", "CCI_CrownCastle_vol_20d", "VIX_Price_zscore_60d__minus__NFCI_zscore_60d", "vix_zscore_10d__minus__EWQ_France_zscore_60d", "AMZN_ret_5d", "ITT_ITTInc_ret_5d", "VIX_Price_zscore_60d__div__hmm_p_stress", "EWL_Switzerland_vol_20d"], "is_new": false}, {"model_id": "v2_h3_GLOBAL_RandomForest_N14", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 3, "n_features": 14, "F1_dir": 0.6721, "F1_UP_FORT": 0.4706, "F1_DOWN_FORT": 0.5314, "train_start": "2007-01-31", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["kalman_innovation__div__kalman_residual", "kalman_innovation__macross__kalman_residual", "PAYX_Paychex_vol_20d", "VIX_Price_zscore_60d__div__vix_max_abs_ret_5d", "TXN_vol_20d__div__heston_xi", "TXN_vol_20d__div__kalman_residual", "CCI_CrownCastle_vol_20d", "VIX_Price_zscore_60d__minus__NFCI_zscore_60d", "vix_zscore_10d__minus__EWQ_France_zscore_60d", "AMZN_ret_5d", "ITT_ITTInc_ret_5d", "VIX_Price_zscore_60d__div__hmm_p_stress", "EWL_Switzerland_vol_20d", "VVIX_ret_20d"], "is_new": false}, {"model_id": "v2_h3_GLOBAL_LogisticRegression_N5", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 3, "n_features": 5, "F1_dir": 0.5655, "F1_UP_FORT": 0.2104, "F1_DOWN_FORT": 0.4251, "train_start": "2007-01-31", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["kalman_innovation__div__kalman_residual", "kalman_innovation__macross__kalman_residual", "PAYX_Paychex_vol_20d", "VIX_Price_zscore_60d__div__vix_max_abs_ret_5d", "TXN_vol_20d__div__heston_xi"], "is_new": false}, {"model_id": "v2_h3_GLOBAL_LogisticRegression_N6", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 3, "n_features": 6, "F1_dir": 0.57, "F1_UP_FORT": 0.1871, "F1_DOWN_FORT": 0.4307, "train_start": "2007-01-31", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["kalman_innovation__div__kalman_residual", "kalman_innovation__macross__kalman_residual", "PAYX_Paychex_vol_20d", "VIX_Price_zscore_60d__div__vix_max_abs_ret_5d", "TXN_vol_20d__div__heston_xi", "TXN_vol_20d__div__kalman_residual"], "is_new": false}, {"model_id": "v2_h3_GLOBAL_LogisticRegression_N7", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 3, "n_features": 7, "F1_dir": 0.5716, "F1_UP_FORT": 0.1839, "F1_DOWN_FORT": 0.4498, "train_start": "2007-01-31", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["kalman_innovation__div__kalman_residual", "kalman_innovation__macross__kalman_residual", "PAYX_Paychex_vol_20d", "VIX_Price_zscore_60d__div__vix_max_abs_ret_5d", "TXN_vol_20d__div__heston_xi", "TXN_vol_20d__div__kalman_residual", "CCI_CrownCastle_vol_20d"], "is_new": false}, {"model_id": "v2_h3_GLOBAL_LogisticRegression_N8", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 3, "n_features": 8, "F1_dir": 0.556, "F1_UP_FORT": 0.2895, "F1_DOWN_FORT": 0.4389, "train_start": "2007-01-31", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["kalman_innovation__div__kalman_residual", "kalman_innovation__macross__kalman_residual", "PAYX_Paychex_vol_20d", "VIX_Price_zscore_60d__div__vix_max_abs_ret_5d", "TXN_vol_20d__div__heston_xi", "TXN_vol_20d__div__kalman_residual", "CCI_CrownCastle_vol_20d", "VIX_Price_zscore_60d__minus__NFCI_zscore_60d"], "is_new": false}, {"model_id": "v2_h3_GLOBAL_LogisticRegression_N9", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 3, "n_features": 9, "F1_dir": 0.5658, "F1_UP_FORT": 0.2676, "F1_DOWN_FORT": 0.4424, "train_start": "2007-01-31", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["kalman_innovation__div__kalman_residual", "kalman_innovation__macross__kalman_residual", "PAYX_Paychex_vol_20d", "VIX_Price_zscore_60d__div__vix_max_abs_ret_5d", "TXN_vol_20d__div__heston_xi", "TXN_vol_20d__div__kalman_residual", "CCI_CrownCastle_vol_20d", "VIX_Price_zscore_60d__minus__NFCI_zscore_60d", "vix_zscore_10d__minus__EWQ_France_zscore_60d"], "is_new": false}, {"model_id": "v2_h3_GLOBAL_LogisticRegression_N10", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 3, "n_features": 10, "F1_dir": 0.5642, "F1_UP_FORT": 0.284, "F1_DOWN_FORT": 0.4334, "train_start": "2007-01-31", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["kalman_innovation__div__kalman_residual", "kalman_innovation__macross__kalman_residual", "PAYX_Paychex_vol_20d", "VIX_Price_zscore_60d__div__vix_max_abs_ret_5d", "TXN_vol_20d__div__heston_xi", "TXN_vol_20d__div__kalman_residual", "CCI_CrownCastle_vol_20d", "VIX_Price_zscore_60d__minus__NFCI_zscore_60d", "vix_zscore_10d__minus__EWQ_France_zscore_60d", "AMZN_ret_5d"], "is_new": false}, {"model_id": "v2_h3_GLOBAL_LogisticRegression_N11", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 3, "n_features": 11, "F1_dir": 0.5718, "F1_UP_FORT": 0.2832, "F1_DOWN_FORT": 0.4258, "train_start": "2007-01-31", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["kalman_innovation__div__kalman_residual", "kalman_innovation__macross__kalman_residual", "PAYX_Paychex_vol_20d", "VIX_Price_zscore_60d__div__vix_max_abs_ret_5d", "TXN_vol_20d__div__heston_xi", "TXN_vol_20d__div__kalman_residual", "CCI_CrownCastle_vol_20d", "VIX_Price_zscore_60d__minus__NFCI_zscore_60d", "vix_zscore_10d__minus__EWQ_France_zscore_60d", "AMZN_ret_5d", "ITT_ITTInc_ret_5d"], "is_new": false}, {"model_id": "v2_h3_GLOBAL_LogisticRegression_N12", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 3, "n_features": 12, "F1_dir": 0.5575, "F1_UP_FORT": 0.2947, "F1_DOWN_FORT": 0.428, "train_start": "2007-01-31", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["kalman_innovation__div__kalman_residual", "kalman_innovation__macross__kalman_residual", "PAYX_Paychex_vol_20d", "VIX_Price_zscore_60d__div__vix_max_abs_ret_5d", "TXN_vol_20d__div__heston_xi", "TXN_vol_20d__div__kalman_residual", "CCI_CrownCastle_vol_20d", "VIX_Price_zscore_60d__minus__NFCI_zscore_60d", "vix_zscore_10d__minus__EWQ_France_zscore_60d", "AMZN_ret_5d", "ITT_ITTInc_ret_5d", "VIX_Price_zscore_60d__div__hmm_p_stress"], "is_new": false}, {"model_id": "v2_h3_GLOBAL_LogisticRegression_N13", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 3, "n_features": 13, "F1_dir": 0.5669, "F1_UP_FORT": 0.3178, "F1_DOWN_FORT": 0.4264, "train_start": "2007-01-31", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["kalman_innovation__div__kalman_residual", "kalman_innovation__macross__kalman_residual", "PAYX_Paychex_vol_20d", "VIX_Price_zscore_60d__div__vix_max_abs_ret_5d", "TXN_vol_20d__div__heston_xi", "TXN_vol_20d__div__kalman_residual", "CCI_CrownCastle_vol_20d", "VIX_Price_zscore_60d__minus__NFCI_zscore_60d", "vix_zscore_10d__minus__EWQ_France_zscore_60d", "AMZN_ret_5d", "ITT_ITTInc_ret_5d", "VIX_Price_zscore_60d__div__hmm_p_stress", "EWL_Switzerland_vol_20d"], "is_new": false}, {"model_id": "v2_h3_GLOBAL_LogisticRegression_N14", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 3, "n_features": 14, "F1_dir": 0.5567, "F1_UP_FORT": 0.3034, "F1_DOWN_FORT": 0.4138, "train_start": "2007-01-31", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["kalman_innovation__div__kalman_residual", "kalman_innovation__macross__kalman_residual", "PAYX_Paychex_vol_20d", "VIX_Price_zscore_60d__div__vix_max_abs_ret_5d", "TXN_vol_20d__div__heston_xi", "TXN_vol_20d__div__kalman_residual", "CCI_CrownCastle_vol_20d", "VIX_Price_zscore_60d__minus__NFCI_zscore_60d", "vix_zscore_10d__minus__EWQ_France_zscore_60d", "AMZN_ret_5d", "ITT_ITTInc_ret_5d", "VIX_Price_zscore_60d__div__hmm_p_stress", "EWL_Switzerland_vol_20d", "VVIX_ret_20d"], "is_new": false}, {"model_id": "v2_h3_GLOBAL_LogisticRegression_N15", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 3, "n_features": 15, "F1_dir": 0.5529, "F1_UP_FORT": 0.3216, "F1_DOWN_FORT": 0.4166, "train_start": "2007-01-31", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["kalman_innovation__div__kalman_residual", "kalman_innovation__macross__kalman_residual", "PAYX_Paychex_vol_20d", "VIX_Price_zscore_60d__div__vix_max_abs_ret_5d", "TXN_vol_20d__div__heston_xi", "TXN_vol_20d__div__kalman_residual", "CCI_CrownCastle_vol_20d", "VIX_Price_zscore_60d__minus__NFCI_zscore_60d", "vix_zscore_10d__minus__EWQ_France_zscore_60d", "AMZN_ret_5d", "ITT_ITTInc_ret_5d", "VIX_Price_zscore_60d__div__hmm_p_stress", "EWL_Switzerland_vol_20d", "VVIX_ret_20d", "vix_vol_of_vol_10d__div__kalman_residual"], "is_new": false}, {"model_id": "v2_h3_GLOBAL_LogisticRegression_N16", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 3, "n_features": 16, "F1_dir": 0.5529, "F1_UP_FORT": 0.3236, "F1_DOWN_FORT": 0.4144, "train_start": "2007-01-31", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["kalman_innovation__div__kalman_residual", "kalman_innovation__macross__kalman_residual", "PAYX_Paychex_vol_20d", "VIX_Price_zscore_60d__div__vix_max_abs_ret_5d", "TXN_vol_20d__div__heston_xi", "TXN_vol_20d__div__kalman_residual", "CCI_CrownCastle_vol_20d", "VIX_Price_zscore_60d__minus__NFCI_zscore_60d", "vix_zscore_10d__minus__EWQ_France_zscore_60d", "AMZN_ret_5d", "ITT_ITTInc_ret_5d", "VIX_Price_zscore_60d__div__hmm_p_stress", "EWL_Switzerland_vol_20d", "VVIX_ret_20d", "vix_vol_of_vol_10d__div__kalman_residual", "heston_xi__div__kalman_residual"], "is_new": false}, {"model_id": "v2_h3_GLOBAL_LogisticRegression_N17", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 3, "n_features": 17, "F1_dir": 0.5558, "F1_UP_FORT": 0.3255, "F1_DOWN_FORT": 0.4186, "train_start": "2007-01-31", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["kalman_innovation__div__kalman_residual", "kalman_innovation__macross__kalman_residual", "PAYX_Paychex_vol_20d", "VIX_Price_zscore_60d__div__vix_max_abs_ret_5d", "TXN_vol_20d__div__heston_xi", "TXN_vol_20d__div__kalman_residual", "CCI_CrownCastle_vol_20d", "VIX_Price_zscore_60d__minus__NFCI_zscore_60d", "vix_zscore_10d__minus__EWQ_France_zscore_60d", "AMZN_ret_5d", "ITT_ITTInc_ret_5d", "VIX_Price_zscore_60d__div__hmm_p_stress", "EWL_Switzerland_vol_20d", "VVIX_ret_20d", "vix_vol_of_vol_10d__div__kalman_residual", "heston_xi__div__kalman_residual", "spx_momentum_3d"], "is_new": false}, {"model_id": "v2_h3_GLOBAL_LogisticRegression_N18", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 3, "n_features": 18, "F1_dir": 0.5594, "F1_UP_FORT": 0.3196, "F1_DOWN_FORT": 0.4327, "train_start": "2007-01-31", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["kalman_innovation__div__kalman_residual", "kalman_innovation__macross__kalman_residual", "PAYX_Paychex_vol_20d", "VIX_Price_zscore_60d__div__vix_max_abs_ret_5d", "TXN_vol_20d__div__heston_xi", "TXN_vol_20d__div__kalman_residual", "CCI_CrownCastle_vol_20d", "VIX_Price_zscore_60d__minus__NFCI_zscore_60d", "vix_zscore_10d__minus__EWQ_France_zscore_60d", "AMZN_ret_5d", "ITT_ITTInc_ret_5d", "VIX_Price_zscore_60d__div__hmm_p_stress", "EWL_Switzerland_vol_20d", "VVIX_ret_20d", "vix_vol_of_vol_10d__div__kalman_residual", "heston_xi__div__kalman_residual", "spx_momentum_3d", "vix_zscore_10d__zrel__XLB_Materials_zscore_60d"], "is_new": false}, {"model_id": "v2_h3_GLOBAL_LogisticRegression_N19", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 3, "n_features": 19, "F1_dir": 0.5616, "F1_UP_FORT": 0.3265, "F1_DOWN_FORT": 0.4408, "train_start": "2007-01-31", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["kalman_innovation__div__kalman_residual", "kalman_innovation__macross__kalman_residual", "PAYX_Paychex_vol_20d", "VIX_Price_zscore_60d__div__vix_max_abs_ret_5d", "TXN_vol_20d__div__heston_xi", "TXN_vol_20d__div__kalman_residual", "CCI_CrownCastle_vol_20d", "VIX_Price_zscore_60d__minus__NFCI_zscore_60d", "vix_zscore_10d__minus__EWQ_France_zscore_60d", "AMZN_ret_5d", "ITT_ITTInc_ret_5d", "VIX_Price_zscore_60d__div__hmm_p_stress", "EWL_Switzerland_vol_20d", "VVIX_ret_20d", "vix_vol_of_vol_10d__div__kalman_residual", "heston_xi__div__kalman_residual", "spx_momentum_3d", "vix_zscore_10d__zrel__XLB_Materials_zscore_60d", "VIX_Price_zscore_60d__ret5x__HangSeng_HK_ret_5d"], "is_new": false}, {"model_id": "v2_h5_CALM_XGBoost_N5", "algo": "XGBoost", "regime": "CALM", "horizon": 5, "n_features": 5, "F1_dir": 0.5169, "F1_UP_FORT": 0.3396, "F1_DOWN_FORT": 0.3876, "train_start": "2000-11-15", "sampler": "SMOTE", "best_params": "{}", "features": ["VIX_Price_zscore_60d__minus__HON_Honeywell_zscore_60d", "NFCI_ret_5d__div__NVDA_vol_20d", "vix_zscore_10d__ret5x__SO_SouthernCo_ret_1d", "VIX_Price_zscore_60d__div__VRP", "vix_zscore_10d__div__spx_abs_ret_max_5d"], "is_new": false}, {"model_id": "v2_h5_CALM_XGBoost_N8", "algo": "XGBoost", "regime": "CALM", "horizon": 5, "n_features": 8, "F1_dir": 0.5083, "F1_UP_FORT": 0.2712, "F1_DOWN_FORT": 0.3103, "train_start": "2000-11-15", "sampler": "SMOTE", "best_params": "{}", "features": ["VIX_Price_zscore_60d__minus__HON_Honeywell_zscore_60d", "NFCI_ret_5d__div__NVDA_vol_20d", "vix_zscore_10d__ret5x__SO_SouthernCo_ret_1d", "VIX_Price_zscore_60d__div__VRP", "vix_zscore_10d__div__spx_abs_ret_max_5d", "EWY_Korea_ret_20d", "EWQ_France_zscore_60d", "LLY_zscore_60d"], "is_new": false}, {"model_id": "v2_h5_CALM_XGBoost_N9", "algo": "XGBoost", "regime": "CALM", "horizon": 5, "n_features": 9, "F1_dir": 0.5255, "F1_UP_FORT": 0.2075, "F1_DOWN_FORT": 0.381, "train_start": "2000-11-15", "sampler": "SMOTE", "best_params": "{}", "features": ["VIX_Price_zscore_60d__minus__HON_Honeywell_zscore_60d", "NFCI_ret_5d__div__NVDA_vol_20d", "vix_zscore_10d__ret5x__SO_SouthernCo_ret_1d", "VIX_Price_zscore_60d__div__VRP", "vix_zscore_10d__div__spx_abs_ret_max_5d", "EWY_Korea_ret_20d", "EWQ_France_zscore_60d", "LLY_zscore_60d", "EWC_Canada_zscore_60d"], "is_new": false}, {"model_id": "v2_h5_CALM_XGBoost_N10", "algo": "XGBoost", "regime": "CALM", "horizon": 5, "n_features": 10, "F1_dir": 0.5257, "F1_UP_FORT": 0.2373, "F1_DOWN_FORT": 0.3846, "train_start": "2000-11-15", "sampler": "SMOTE", "best_params": "{}", "features": ["VIX_Price_zscore_60d__minus__HON_Honeywell_zscore_60d", "NFCI_ret_5d__div__NVDA_vol_20d", "vix_zscore_10d__ret5x__SO_SouthernCo_ret_1d", "VIX_Price_zscore_60d__div__VRP", "vix_zscore_10d__div__spx_abs_ret_max_5d", "EWY_Korea_ret_20d", "EWQ_France_zscore_60d", "LLY_zscore_60d", "EWC_Canada_zscore_60d", "Michigan_Sentiment_zscore_60d__zrel__AMZN_zscore_60d"], "is_new": false}, {"model_id": "v2_h5_CALM_XGBoost_N11", "algo": "XGBoost", "regime": "CALM", "horizon": 5, "n_features": 11, "F1_dir": 0.5301, "F1_UP_FORT": 0.193, "F1_DOWN_FORT": 0.3511, "train_start": "2000-11-15", "sampler": "SMOTE", "best_params": "{}", "features": ["VIX_Price_zscore_60d__minus__HON_Honeywell_zscore_60d", "NFCI_ret_5d__div__NVDA_vol_20d", "vix_zscore_10d__ret5x__SO_SouthernCo_ret_1d", "VIX_Price_zscore_60d__div__VRP", "vix_zscore_10d__div__spx_abs_ret_max_5d", "EWY_Korea_ret_20d", "EWQ_France_zscore_60d", "LLY_zscore_60d", "EWC_Canada_zscore_60d", "Michigan_Sentiment_zscore_60d__zrel__AMZN_zscore_60d", "NFCI_ret_5d__div__SO_SouthernCo_ret_1d"], "is_new": false}, {"model_id": "v2_h5_CALM_XGBoost_N12", "algo": "XGBoost", "regime": "CALM", "horizon": 5, "n_features": 12, "F1_dir": 0.5039, "F1_UP_FORT": 0.1042, "F1_DOWN_FORT": 0.4088, "train_start": "2000-11-15", "sampler": "SMOTE", "best_params": "{}", "features": ["VIX_Price_zscore_60d__minus__HON_Honeywell_zscore_60d", "NFCI_ret_5d__div__NVDA_vol_20d", "vix_zscore_10d__ret5x__SO_SouthernCo_ret_1d", "VIX_Price_zscore_60d__div__VRP", "vix_zscore_10d__div__spx_abs_ret_max_5d", "EWY_Korea_ret_20d", "EWQ_France_zscore_60d", "LLY_zscore_60d", "EWC_Canada_zscore_60d", "Michigan_Sentiment_zscore_60d__zrel__AMZN_zscore_60d", "NFCI_ret_5d__div__SO_SouthernCo_ret_1d", "XLY_Disc_vol_20d"], "is_new": false}, {"model_id": "v2_h5_CALM_XGBoost_N13", "algo": "XGBoost", "regime": "CALM", "horizon": 5, "n_features": 13, "F1_dir": 0.5517, "F1_UP_FORT": 0.1818, "F1_DOWN_FORT": 0.4088, "train_start": "2000-11-15", "sampler": "SMOTE", "best_params": "{}", "features": ["VIX_Price_zscore_60d__minus__HON_Honeywell_zscore_60d", "NFCI_ret_5d__div__NVDA_vol_20d", "vix_zscore_10d__ret5x__SO_SouthernCo_ret_1d", "VIX_Price_zscore_60d__div__VRP", "vix_zscore_10d__div__spx_abs_ret_max_5d", "EWY_Korea_ret_20d", "EWQ_France_zscore_60d", "LLY_zscore_60d", "EWC_Canada_zscore_60d", "Michigan_Sentiment_zscore_60d__zrel__AMZN_zscore_60d", "NFCI_ret_5d__div__SO_SouthernCo_ret_1d", "XLY_Disc_vol_20d", "ASML_ASML_ret_5d__div__spx_momentum_3d"], "is_new": false}, {"model_id": "v2_h5_CALM_XGBoost_N14", "algo": "XGBoost", "regime": "CALM", "horizon": 5, "n_features": 14, "F1_dir": 0.5602, "F1_UP_FORT": 0.1136, "F1_DOWN_FORT": 0.3971, "train_start": "2000-11-15", "sampler": "SMOTE", "best_params": "{}", "features": ["VIX_Price_zscore_60d__minus__HON_Honeywell_zscore_60d", "NFCI_ret_5d__div__NVDA_vol_20d", "vix_zscore_10d__ret5x__SO_SouthernCo_ret_1d", "VIX_Price_zscore_60d__div__VRP", "vix_zscore_10d__div__spx_abs_ret_max_5d", "EWY_Korea_ret_20d", "EWQ_France_zscore_60d", "LLY_zscore_60d", "EWC_Canada_zscore_60d", "Michigan_Sentiment_zscore_60d__zrel__AMZN_zscore_60d", "NFCI_ret_5d__div__SO_SouthernCo_ret_1d", "XLY_Disc_vol_20d", "ASML_ASML_ret_5d__div__spx_momentum_3d", "NFCI_ret_5d"], "is_new": false}, {"model_id": "v2_h5_CALM_XGBoost_N15", "algo": "XGBoost", "regime": "CALM", "horizon": 5, "n_features": 15, "F1_dir": 0.5723, "F1_UP_FORT": 0.1616, "F1_DOWN_FORT": 0.3898, "train_start": "2000-11-15", "sampler": "SMOTE", "best_params": "{}", "features": ["VIX_Price_zscore_60d__minus__HON_Honeywell_zscore_60d", "NFCI_ret_5d__div__NVDA_vol_20d", "vix_zscore_10d__ret5x__SO_SouthernCo_ret_1d", "VIX_Price_zscore_60d__div__VRP", "vix_zscore_10d__div__spx_abs_ret_max_5d", "EWY_Korea_ret_20d", "EWQ_France_zscore_60d", "LLY_zscore_60d", "EWC_Canada_zscore_60d", "Michigan_Sentiment_zscore_60d__zrel__AMZN_zscore_60d", "NFCI_ret_5d__div__SO_SouthernCo_ret_1d", "XLY_Disc_vol_20d", "ASML_ASML_ret_5d__div__spx_momentum_3d", "NFCI_ret_5d", "Nikkei_Japan_zscore_60d__prod__AMZN_zscore_60d"], "is_new": false}, {"model_id": "v2_h5_CALM_XGBoost_N16", "algo": "XGBoost", "regime": "CALM", "horizon": 5, "n_features": 16, "F1_dir": 0.5637, "F1_UP_FORT": 0.1684, "F1_DOWN_FORT": 0.3361, "train_start": "2000-11-15", "sampler": "SMOTE", "best_params": "{}", "features": ["VIX_Price_zscore_60d__minus__HON_Honeywell_zscore_60d", "NFCI_ret_5d__div__NVDA_vol_20d", "vix_zscore_10d__ret5x__SO_SouthernCo_ret_1d", "VIX_Price_zscore_60d__div__VRP", "vix_zscore_10d__div__spx_abs_ret_max_5d", "EWY_Korea_ret_20d", "EWQ_France_zscore_60d", "LLY_zscore_60d", "EWC_Canada_zscore_60d", "Michigan_Sentiment_zscore_60d__zrel__AMZN_zscore_60d", "NFCI_ret_5d__div__SO_SouthernCo_ret_1d", "XLY_Disc_vol_20d", "ASML_ASML_ret_5d__div__spx_momentum_3d", "NFCI_ret_5d", "Nikkei_Japan_zscore_60d__prod__AMZN_zscore_60d", "T10Y2Y_Spread_ret_5d"], "is_new": false}, {"model_id": "v2_h5_CALM_XGBoost_N17", "algo": "XGBoost", "regime": "CALM", "horizon": 5, "n_features": 17, "F1_dir": 0.5904, "F1_UP_FORT": 0.1765, "F1_DOWN_FORT": 0.3299, "train_start": "2000-11-15", "sampler": "SMOTE", "best_params": "{}", "features": ["VIX_Price_zscore_60d__minus__HON_Honeywell_zscore_60d", "NFCI_ret_5d__div__NVDA_vol_20d", "vix_zscore_10d__ret5x__SO_SouthernCo_ret_1d", "VIX_Price_zscore_60d__div__VRP", "vix_zscore_10d__div__spx_abs_ret_max_5d", "EWY_Korea_ret_20d", "EWQ_France_zscore_60d", "LLY_zscore_60d", "EWC_Canada_zscore_60d", "Michigan_Sentiment_zscore_60d__zrel__AMZN_zscore_60d", "NFCI_ret_5d__div__SO_SouthernCo_ret_1d", "XLY_Disc_vol_20d", "ASML_ASML_ret_5d__div__spx_momentum_3d", "NFCI_ret_5d", "Nikkei_Japan_zscore_60d__prod__AMZN_zscore_60d", "T10Y2Y_Spread_ret_5d", "TM_Telephone_vol_20d"], "is_new": false}, {"model_id": "v2_h5_CALM_XGBoost_N18", "algo": "XGBoost", "regime": "CALM", "horizon": 5, "n_features": 18, "F1_dir": 0.5626, "F1_UP_FORT": 0.1538, "F1_DOWN_FORT": 0.2692, "train_start": "2000-11-15", "sampler": "SMOTE", "best_params": "{}", "features": ["VIX_Price_zscore_60d__minus__HON_Honeywell_zscore_60d", "NFCI_ret_5d__div__NVDA_vol_20d", "vix_zscore_10d__ret5x__SO_SouthernCo_ret_1d", "VIX_Price_zscore_60d__div__VRP", "vix_zscore_10d__div__spx_abs_ret_max_5d", "EWY_Korea_ret_20d", "EWQ_France_zscore_60d", "LLY_zscore_60d", "EWC_Canada_zscore_60d", "Michigan_Sentiment_zscore_60d__zrel__AMZN_zscore_60d", "NFCI_ret_5d__div__SO_SouthernCo_ret_1d", "XLY_Disc_vol_20d", "ASML_ASML_ret_5d__div__spx_momentum_3d", "NFCI_ret_5d", "Nikkei_Japan_zscore_60d__prod__AMZN_zscore_60d", "T10Y2Y_Spread_ret_5d", "TM_Telephone_vol_20d", "vix_zscore_10d__minus__HON_Honeywell_zscore_60d"], "is_new": false}, {"model_id": "v2_h5_CALM_LightGBM_N9", "algo": "LightGBM", "regime": "CALM", "horizon": 5, "n_features": 9, "F1_dir": 0.5577, "F1_UP_FORT": 0.25, "F1_DOWN_FORT": 0.4034, "train_start": "2000-11-15", "sampler": "SMOTE", "best_params": "{}", "features": ["VIX_Price_zscore_60d__minus__HON_Honeywell_zscore_60d", "NFCI_ret_5d__div__NVDA_vol_20d", "vix_zscore_10d__ret5x__SO_SouthernCo_ret_1d", "VIX_Price_zscore_60d__div__VRP", "vix_zscore_10d__div__spx_abs_ret_max_5d", "EWY_Korea_ret_20d", "EWQ_France_zscore_60d", "LLY_zscore_60d", "EWC_Canada_zscore_60d"], "is_new": false}, {"model_id": "v2_h5_CALM_LightGBM_N11", "algo": "LightGBM", "regime": "CALM", "horizon": 5, "n_features": 11, "F1_dir": 0.5241, "F1_UP_FORT": 0.2479, "F1_DOWN_FORT": 0.3415, "train_start": "2000-11-15", "sampler": "SMOTE", "best_params": "{}", "features": ["VIX_Price_zscore_60d__minus__HON_Honeywell_zscore_60d", "NFCI_ret_5d__div__NVDA_vol_20d", "vix_zscore_10d__ret5x__SO_SouthernCo_ret_1d", "VIX_Price_zscore_60d__div__VRP", "vix_zscore_10d__div__spx_abs_ret_max_5d", "EWY_Korea_ret_20d", "EWQ_France_zscore_60d", "LLY_zscore_60d", "EWC_Canada_zscore_60d", "Michigan_Sentiment_zscore_60d__zrel__AMZN_zscore_60d", "NFCI_ret_5d__div__SO_SouthernCo_ret_1d"], "is_new": false}, {"model_id": "v2_h5_CALM_LightGBM_N12", "algo": "LightGBM", "regime": "CALM", "horizon": 5, "n_features": 12, "F1_dir": 0.5472, "F1_UP_FORT": 0.0833, "F1_DOWN_FORT": 0.3438, "train_start": "2000-11-15", "sampler": "SMOTE", "best_params": "{}", "features": ["VIX_Price_zscore_60d__minus__HON_Honeywell_zscore_60d", "NFCI_ret_5d__div__NVDA_vol_20d", "vix_zscore_10d__ret5x__SO_SouthernCo_ret_1d", "VIX_Price_zscore_60d__div__VRP", "vix_zscore_10d__div__spx_abs_ret_max_5d", "EWY_Korea_ret_20d", "EWQ_France_zscore_60d", "LLY_zscore_60d", "EWC_Canada_zscore_60d", "Michigan_Sentiment_zscore_60d__zrel__AMZN_zscore_60d", "NFCI_ret_5d__div__SO_SouthernCo_ret_1d", "XLY_Disc_vol_20d"], "is_new": false}, {"model_id": "v2_h5_CALM_LightGBM_N13", "algo": "LightGBM", "regime": "CALM", "horizon": 5, "n_features": 13, "F1_dir": 0.5546, "F1_UP_FORT": 0.1458, "F1_DOWN_FORT": 0.3969, "train_start": "2000-11-15", "sampler": "SMOTE", "best_params": "{}", "features": ["VIX_Price_zscore_60d__minus__HON_Honeywell_zscore_60d", "NFCI_ret_5d__div__NVDA_vol_20d", "vix_zscore_10d__ret5x__SO_SouthernCo_ret_1d", "VIX_Price_zscore_60d__div__VRP", "vix_zscore_10d__div__spx_abs_ret_max_5d", "EWY_Korea_ret_20d", "EWQ_France_zscore_60d", "LLY_zscore_60d", "EWC_Canada_zscore_60d", "Michigan_Sentiment_zscore_60d__zrel__AMZN_zscore_60d", "NFCI_ret_5d__div__SO_SouthernCo_ret_1d", "XLY_Disc_vol_20d", "ASML_ASML_ret_5d__div__spx_momentum_3d"], "is_new": false}, {"model_id": "v2_h5_CALM_LightGBM_N14", "algo": "LightGBM", "regime": "CALM", "horizon": 5, "n_features": 14, "F1_dir": 0.5118, "F1_UP_FORT": 0.1489, "F1_DOWN_FORT": 0.3538, "train_start": "2000-11-15", "sampler": "SMOTE", "best_params": "{}", "features": ["VIX_Price_zscore_60d__minus__HON_Honeywell_zscore_60d", "NFCI_ret_5d__div__NVDA_vol_20d", "vix_zscore_10d__ret5x__SO_SouthernCo_ret_1d", "VIX_Price_zscore_60d__div__VRP", "vix_zscore_10d__div__spx_abs_ret_max_5d", "EWY_Korea_ret_20d", "EWQ_France_zscore_60d", "LLY_zscore_60d", "EWC_Canada_zscore_60d", "Michigan_Sentiment_zscore_60d__zrel__AMZN_zscore_60d", "NFCI_ret_5d__div__SO_SouthernCo_ret_1d", "XLY_Disc_vol_20d", "ASML_ASML_ret_5d__div__spx_momentum_3d", "NFCI_ret_5d"], "is_new": false}, {"model_id": "v2_h5_CALM_LightGBM_N15", "algo": "LightGBM", "regime": "CALM", "horizon": 5, "n_features": 15, "F1_dir": 0.5469, "F1_UP_FORT": 0.1224, "F1_DOWN_FORT": 0.3359, "train_start": "2000-11-15", "sampler": "SMOTE", "best_params": "{}", "features": ["VIX_Price_zscore_60d__minus__HON_Honeywell_zscore_60d", "NFCI_ret_5d__div__NVDA_vol_20d", "vix_zscore_10d__ret5x__SO_SouthernCo_ret_1d", "VIX_Price_zscore_60d__div__VRP", "vix_zscore_10d__div__spx_abs_ret_max_5d", "EWY_Korea_ret_20d", "EWQ_France_zscore_60d", "LLY_zscore_60d", "EWC_Canada_zscore_60d", "Michigan_Sentiment_zscore_60d__zrel__AMZN_zscore_60d", "NFCI_ret_5d__div__SO_SouthernCo_ret_1d", "XLY_Disc_vol_20d", "ASML_ASML_ret_5d__div__spx_momentum_3d", "NFCI_ret_5d", "Nikkei_Japan_zscore_60d__prod__AMZN_zscore_60d"], "is_new": false}, {"model_id": "v2_h5_CALM_LightGBM_N16", "algo": "LightGBM", "regime": "CALM", "horizon": 5, "n_features": 16, "F1_dir": 0.5342, "F1_UP_FORT": 0.1042, "F1_DOWN_FORT": 0.3443, "train_start": "2000-11-15", "sampler": "SMOTE", "best_params": "{}", "features": ["VIX_Price_zscore_60d__minus__HON_Honeywell_zscore_60d", "NFCI_ret_5d__div__NVDA_vol_20d", "vix_zscore_10d__ret5x__SO_SouthernCo_ret_1d", "VIX_Price_zscore_60d__div__VRP", "vix_zscore_10d__div__spx_abs_ret_max_5d", "EWY_Korea_ret_20d", "EWQ_France_zscore_60d", "LLY_zscore_60d", "EWC_Canada_zscore_60d", "Michigan_Sentiment_zscore_60d__zrel__AMZN_zscore_60d", "NFCI_ret_5d__div__SO_SouthernCo_ret_1d", "XLY_Disc_vol_20d", "ASML_ASML_ret_5d__div__spx_momentum_3d", "NFCI_ret_5d", "Nikkei_Japan_zscore_60d__prod__AMZN_zscore_60d", "T10Y2Y_Spread_ret_5d"], "is_new": false}, {"model_id": "v2_h5_CALM_LightGBM_N17", "algo": "LightGBM", "regime": "CALM", "horizon": 5, "n_features": 17, "F1_dir": 0.5722, "F1_UP_FORT": 0.125, "F1_DOWN_FORT": 0.3168, "train_start": "2000-11-15", "sampler": "SMOTE", "best_params": "{}", "features": ["VIX_Price_zscore_60d__minus__HON_Honeywell_zscore_60d", "NFCI_ret_5d__div__NVDA_vol_20d", "vix_zscore_10d__ret5x__SO_SouthernCo_ret_1d", "VIX_Price_zscore_60d__div__VRP", "vix_zscore_10d__div__spx_abs_ret_max_5d", "EWY_Korea_ret_20d", "EWQ_France_zscore_60d", "LLY_zscore_60d", "EWC_Canada_zscore_60d", "Michigan_Sentiment_zscore_60d__zrel__AMZN_zscore_60d", "NFCI_ret_5d__div__SO_SouthernCo_ret_1d", "XLY_Disc_vol_20d", "ASML_ASML_ret_5d__div__spx_momentum_3d", "NFCI_ret_5d", "Nikkei_Japan_zscore_60d__prod__AMZN_zscore_60d", "T10Y2Y_Spread_ret_5d", "TM_Telephone_vol_20d"], "is_new": false}, {"model_id": "v2_h5_CALM_LightGBM_N18", "algo": "LightGBM", "regime": "CALM", "horizon": 5, "n_features": 18, "F1_dir": 0.5207, "F1_UP_FORT": 0.1333, "F1_DOWN_FORT": 0.2752, "train_start": "2000-11-15", "sampler": "SMOTE", "best_params": "{}", "features": ["VIX_Price_zscore_60d__minus__HON_Honeywell_zscore_60d", "NFCI_ret_5d__div__NVDA_vol_20d", "vix_zscore_10d__ret5x__SO_SouthernCo_ret_1d", "VIX_Price_zscore_60d__div__VRP", "vix_zscore_10d__div__spx_abs_ret_max_5d", "EWY_Korea_ret_20d", "EWQ_France_zscore_60d", "LLY_zscore_60d", "EWC_Canada_zscore_60d", "Michigan_Sentiment_zscore_60d__zrel__AMZN_zscore_60d", "NFCI_ret_5d__div__SO_SouthernCo_ret_1d", "XLY_Disc_vol_20d", "ASML_ASML_ret_5d__div__spx_momentum_3d", "NFCI_ret_5d", "Nikkei_Japan_zscore_60d__prod__AMZN_zscore_60d", "T10Y2Y_Spread_ret_5d", "TM_Telephone_vol_20d", "vix_zscore_10d__minus__HON_Honeywell_zscore_60d"], "is_new": false}, {"model_id": "v2_h5_CALM_GradientBoosting_N5", "algo": "GradientBoosting", "regime": "CALM", "horizon": 5, "n_features": 5, "F1_dir": 0.5263, "F1_UP_FORT": 0.3273, "F1_DOWN_FORT": 0.4194, "train_start": "2000-11-15", "sampler": "SMOTE", "best_params": "{}", "features": ["VIX_Price_zscore_60d__minus__HON_Honeywell_zscore_60d", "NFCI_ret_5d__div__NVDA_vol_20d", "vix_zscore_10d__ret5x__SO_SouthernCo_ret_1d", "VIX_Price_zscore_60d__div__VRP", "vix_zscore_10d__div__spx_abs_ret_max_5d"], "is_new": false}, {"model_id": "v2_h5_CALM_GradientBoosting_N9", "algo": "GradientBoosting", "regime": "CALM", "horizon": 5, "n_features": 9, "F1_dir": 0.5317, "F1_UP_FORT": 0.2373, "F1_DOWN_FORT": 0.2931, "train_start": "2000-11-15", "sampler": "SMOTE", "best_params": "{}", "features": ["VIX_Price_zscore_60d__minus__HON_Honeywell_zscore_60d", "NFCI_ret_5d__div__NVDA_vol_20d", "vix_zscore_10d__ret5x__SO_SouthernCo_ret_1d", "VIX_Price_zscore_60d__div__VRP", "vix_zscore_10d__div__spx_abs_ret_max_5d", "EWY_Korea_ret_20d", "EWQ_France_zscore_60d", "LLY_zscore_60d", "EWC_Canada_zscore_60d"], "is_new": false}, {"model_id": "v2_h5_CALM_GradientBoosting_N10", "algo": "GradientBoosting", "regime": "CALM", "horizon": 5, "n_features": 10, "F1_dir": 0.5645, "F1_UP_FORT": 0.2783, "F1_DOWN_FORT": 0.3333, "train_start": "2000-11-15", "sampler": "SMOTE", "best_params": "{}", "features": ["VIX_Price_zscore_60d__minus__HON_Honeywell_zscore_60d", "NFCI_ret_5d__div__NVDA_vol_20d", "vix_zscore_10d__ret5x__SO_SouthernCo_ret_1d", "VIX_Price_zscore_60d__div__VRP", "vix_zscore_10d__div__spx_abs_ret_max_5d", "EWY_Korea_ret_20d", "EWQ_France_zscore_60d", "LLY_zscore_60d", "EWC_Canada_zscore_60d", "Michigan_Sentiment_zscore_60d__zrel__AMZN_zscore_60d"], "is_new": false}, {"model_id": "v2_h5_CALM_GradientBoosting_N11", "algo": "GradientBoosting", "regime": "CALM", "horizon": 5, "n_features": 11, "F1_dir": 0.5328, "F1_UP_FORT": 0.2143, "F1_DOWN_FORT": 0.3279, "train_start": "2000-11-15", "sampler": "SMOTE", "best_params": "{}", "features": ["VIX_Price_zscore_60d__minus__HON_Honeywell_zscore_60d", "NFCI_ret_5d__div__NVDA_vol_20d", "vix_zscore_10d__ret5x__SO_SouthernCo_ret_1d", "VIX_Price_zscore_60d__div__VRP", "vix_zscore_10d__div__spx_abs_ret_max_5d", "EWY_Korea_ret_20d", "EWQ_France_zscore_60d", "LLY_zscore_60d", "EWC_Canada_zscore_60d", "Michigan_Sentiment_zscore_60d__zrel__AMZN_zscore_60d", "NFCI_ret_5d__div__SO_SouthernCo_ret_1d"], "is_new": false}, {"model_id": "v2_h5_CALM_GradientBoosting_N12", "algo": "GradientBoosting", "regime": "CALM", "horizon": 5, "n_features": 12, "F1_dir": 0.5474, "F1_UP_FORT": 0.1616, "F1_DOWN_FORT": 0.4328, "train_start": "2000-11-15", "sampler": "SMOTE", "best_params": "{}", "features": ["VIX_Price_zscore_60d__minus__HON_Honeywell_zscore_60d", "NFCI_ret_5d__div__NVDA_vol_20d", "vix_zscore_10d__ret5x__SO_SouthernCo_ret_1d", "VIX_Price_zscore_60d__div__VRP", "vix_zscore_10d__div__spx_abs_ret_max_5d", "EWY_Korea_ret_20d", "EWQ_France_zscore_60d", "LLY_zscore_60d", "EWC_Canada_zscore_60d", "Michigan_Sentiment_zscore_60d__zrel__AMZN_zscore_60d", "NFCI_ret_5d__div__SO_SouthernCo_ret_1d", "XLY_Disc_vol_20d"], "is_new": false}, {"model_id": "v2_h5_CALM_GradientBoosting_N13", "algo": "GradientBoosting", "regime": "CALM", "horizon": 5, "n_features": 13, "F1_dir": 0.5582, "F1_UP_FORT": 0.26, "F1_DOWN_FORT": 0.3651, "train_start": "2000-11-15", "sampler": "SMOTE", "best_params": "{}", "features": ["VIX_Price_zscore_60d__minus__HON_Honeywell_zscore_60d", "NFCI_ret_5d__div__NVDA_vol_20d", "vix_zscore_10d__ret5x__SO_SouthernCo_ret_1d", "VIX_Price_zscore_60d__div__VRP", "vix_zscore_10d__div__spx_abs_ret_max_5d", "EWY_Korea_ret_20d", "EWQ_France_zscore_60d", "LLY_zscore_60d", "EWC_Canada_zscore_60d", "Michigan_Sentiment_zscore_60d__zrel__AMZN_zscore_60d", "NFCI_ret_5d__div__SO_SouthernCo_ret_1d", "XLY_Disc_vol_20d", "ASML_ASML_ret_5d__div__spx_momentum_3d"], "is_new": false}, {"model_id": "v2_h5_CALM_GradientBoosting_N14", "algo": "GradientBoosting", "regime": "CALM", "horizon": 5, "n_features": 14, "F1_dir": 0.5554, "F1_UP_FORT": 0.1935, "F1_DOWN_FORT": 0.4091, "train_start": "2000-11-15", "sampler": "SMOTE", "best_params": "{}", "features": ["VIX_Price_zscore_60d__minus__HON_Honeywell_zscore_60d", "NFCI_ret_5d__div__NVDA_vol_20d", "vix_zscore_10d__ret5x__SO_SouthernCo_ret_1d", "VIX_Price_zscore_60d__div__VRP", "vix_zscore_10d__div__spx_abs_ret_max_5d", "EWY_Korea_ret_20d", "EWQ_France_zscore_60d", "LLY_zscore_60d", "EWC_Canada_zscore_60d", "Michigan_Sentiment_zscore_60d__zrel__AMZN_zscore_60d", "NFCI_ret_5d__div__SO_SouthernCo_ret_1d", "XLY_Disc_vol_20d", "ASML_ASML_ret_5d__div__spx_momentum_3d", "NFCI_ret_5d"], "is_new": false}, {"model_id": "v2_h5_CALM_GradientBoosting_N15", "algo": "GradientBoosting", "regime": "CALM", "horizon": 5, "n_features": 15, "F1_dir": 0.5837, "F1_UP_FORT": 0.1429, "F1_DOWN_FORT": 0.3802, "train_start": "2000-11-15", "sampler": "SMOTE", "best_params": "{}", "features": ["VIX_Price_zscore_60d__minus__HON_Honeywell_zscore_60d", "NFCI_ret_5d__div__NVDA_vol_20d", "vix_zscore_10d__ret5x__SO_SouthernCo_ret_1d", "VIX_Price_zscore_60d__div__VRP", "vix_zscore_10d__div__spx_abs_ret_max_5d", "EWY_Korea_ret_20d", "EWQ_France_zscore_60d", "LLY_zscore_60d", "EWC_Canada_zscore_60d", "Michigan_Sentiment_zscore_60d__zrel__AMZN_zscore_60d", "NFCI_ret_5d__div__SO_SouthernCo_ret_1d", "XLY_Disc_vol_20d", "ASML_ASML_ret_5d__div__spx_momentum_3d", "NFCI_ret_5d", "Nikkei_Japan_zscore_60d__prod__AMZN_zscore_60d"], "is_new": false}, {"model_id": "v2_h5_CALM_GradientBoosting_N16", "algo": "GradientBoosting", "regime": "CALM", "horizon": 5, "n_features": 16, "F1_dir": 0.5657, "F1_UP_FORT": 0.1443, "F1_DOWN_FORT": 0.3361, "train_start": "2000-11-15", "sampler": "SMOTE", "best_params": "{}", "features": ["VIX_Price_zscore_60d__minus__HON_Honeywell_zscore_60d", "NFCI_ret_5d__div__NVDA_vol_20d", "vix_zscore_10d__ret5x__SO_SouthernCo_ret_1d", "VIX_Price_zscore_60d__div__VRP", "vix_zscore_10d__div__spx_abs_ret_max_5d", "EWY_Korea_ret_20d", "EWQ_France_zscore_60d", "LLY_zscore_60d", "EWC_Canada_zscore_60d", "Michigan_Sentiment_zscore_60d__zrel__AMZN_zscore_60d", "NFCI_ret_5d__div__SO_SouthernCo_ret_1d", "XLY_Disc_vol_20d", "ASML_ASML_ret_5d__div__spx_momentum_3d", "NFCI_ret_5d", "Nikkei_Japan_zscore_60d__prod__AMZN_zscore_60d", "T10Y2Y_Spread_ret_5d"], "is_new": false}, {"model_id": "v2_h5_CALM_GradientBoosting_N17", "algo": "GradientBoosting", "regime": "CALM", "horizon": 5, "n_features": 17, "F1_dir": 0.5508, "F1_UP_FORT": 0.1165, "F1_DOWN_FORT": 0.2804, "train_start": "2000-11-15", "sampler": "SMOTE", "best_params": "{}", "features": ["VIX_Price_zscore_60d__minus__HON_Honeywell_zscore_60d", "NFCI_ret_5d__div__NVDA_vol_20d", "vix_zscore_10d__ret5x__SO_SouthernCo_ret_1d", "VIX_Price_zscore_60d__div__VRP", "vix_zscore_10d__div__spx_abs_ret_max_5d", "EWY_Korea_ret_20d", "EWQ_France_zscore_60d", "LLY_zscore_60d", "EWC_Canada_zscore_60d", "Michigan_Sentiment_zscore_60d__zrel__AMZN_zscore_60d", "NFCI_ret_5d__div__SO_SouthernCo_ret_1d", "XLY_Disc_vol_20d", "ASML_ASML_ret_5d__div__spx_momentum_3d", "NFCI_ret_5d", "Nikkei_Japan_zscore_60d__prod__AMZN_zscore_60d", "T10Y2Y_Spread_ret_5d", "TM_Telephone_vol_20d"], "is_new": false}, {"model_id": "v2_h5_CALM_GradientBoosting_N18", "algo": "GradientBoosting", "regime": "CALM", "horizon": 5, "n_features": 18, "F1_dir": 0.5669, "F1_UP_FORT": 0.1333, "F1_DOWN_FORT": 0.339, "train_start": "2000-11-15", "sampler": "SMOTE", "best_params": "{}", "features": ["VIX_Price_zscore_60d__minus__HON_Honeywell_zscore_60d", "NFCI_ret_5d__div__NVDA_vol_20d", "vix_zscore_10d__ret5x__SO_SouthernCo_ret_1d", "VIX_Price_zscore_60d__div__VRP", "vix_zscore_10d__div__spx_abs_ret_max_5d", "EWY_Korea_ret_20d", "EWQ_France_zscore_60d", "LLY_zscore_60d", "EWC_Canada_zscore_60d", "Michigan_Sentiment_zscore_60d__zrel__AMZN_zscore_60d", "NFCI_ret_5d__div__SO_SouthernCo_ret_1d", "XLY_Disc_vol_20d", "ASML_ASML_ret_5d__div__spx_momentum_3d", "NFCI_ret_5d", "Nikkei_Japan_zscore_60d__prod__AMZN_zscore_60d", "T10Y2Y_Spread_ret_5d", "TM_Telephone_vol_20d", "vix_zscore_10d__minus__HON_Honeywell_zscore_60d"], "is_new": false}, {"model_id": "v2_h5_CALM_RandomForest_N7", "algo": "RandomForest", "regime": "CALM", "horizon": 5, "n_features": 7, "F1_dir": 0.5073, "F1_UP_FORT": 0.2264, "F1_DOWN_FORT": 0.3165, "train_start": "2000-11-15", "sampler": "SMOTE", "best_params": "{}", "features": ["VIX_Price_zscore_60d__minus__HON_Honeywell_zscore_60d", "NFCI_ret_5d__div__NVDA_vol_20d", "vix_zscore_10d__ret5x__SO_SouthernCo_ret_1d", "VIX_Price_zscore_60d__div__VRP", "vix_zscore_10d__div__spx_abs_ret_max_5d", "EWY_Korea_ret_20d", "EWQ_France_zscore_60d"], "is_new": false}, {"model_id": "v2_h5_CALM_RandomForest_N8", "algo": "RandomForest", "regime": "CALM", "horizon": 5, "n_features": 8, "F1_dir": 0.5068, "F1_UP_FORT": 0.2308, "F1_DOWN_FORT": 0.3597, "train_start": "2000-11-15", "sampler": "SMOTE", "best_params": "{}", "features": ["VIX_Price_zscore_60d__minus__HON_Honeywell_zscore_60d", "NFCI_ret_5d__div__NVDA_vol_20d", "vix_zscore_10d__ret5x__SO_SouthernCo_ret_1d", "VIX_Price_zscore_60d__div__VRP", "vix_zscore_10d__div__spx_abs_ret_max_5d", "EWY_Korea_ret_20d", "EWQ_France_zscore_60d", "LLY_zscore_60d"], "is_new": false}, {"model_id": "v2_h5_CALM_RandomForest_N9", "algo": "RandomForest", "regime": "CALM", "horizon": 5, "n_features": 9, "F1_dir": 0.5301, "F1_UP_FORT": 0.2, "F1_DOWN_FORT": 0.3438, "train_start": "2000-11-15", "sampler": "SMOTE", "best_params": "{}", "features": ["VIX_Price_zscore_60d__minus__HON_Honeywell_zscore_60d", "NFCI_ret_5d__div__NVDA_vol_20d", "vix_zscore_10d__ret5x__SO_SouthernCo_ret_1d", "VIX_Price_zscore_60d__div__VRP", "vix_zscore_10d__div__spx_abs_ret_max_5d", "EWY_Korea_ret_20d", "EWQ_France_zscore_60d", "LLY_zscore_60d", "EWC_Canada_zscore_60d"], "is_new": false}, {"model_id": "v2_h5_CALM_RandomForest_N10", "algo": "RandomForest", "regime": "CALM", "horizon": 5, "n_features": 10, "F1_dir": 0.5776, "F1_UP_FORT": 0.2883, "F1_DOWN_FORT": 0.3846, "train_start": "2000-11-15", "sampler": "SMOTE", "best_params": "{}", "features": ["VIX_Price_zscore_60d__minus__HON_Honeywell_zscore_60d", "NFCI_ret_5d__div__NVDA_vol_20d", "vix_zscore_10d__ret5x__SO_SouthernCo_ret_1d", "VIX_Price_zscore_60d__div__VRP", "vix_zscore_10d__div__spx_abs_ret_max_5d", "EWY_Korea_ret_20d", "EWQ_France_zscore_60d", "LLY_zscore_60d", "EWC_Canada_zscore_60d", "Michigan_Sentiment_zscore_60d__zrel__AMZN_zscore_60d"], "is_new": false}, {"model_id": "v2_h5_CALM_RandomForest_N11", "algo": "RandomForest", "regime": "CALM", "horizon": 5, "n_features": 11, "F1_dir": 0.5689, "F1_UP_FORT": 0.2727, "F1_DOWN_FORT": 0.3664, "train_start": "2000-11-15", "sampler": "SMOTE", "best_params": "{}", "features": ["VIX_Price_zscore_60d__minus__HON_Honeywell_zscore_60d", "NFCI_ret_5d__div__NVDA_vol_20d", "vix_zscore_10d__ret5x__SO_SouthernCo_ret_1d", "VIX_Price_zscore_60d__div__VRP", "vix_zscore_10d__div__spx_abs_ret_max_5d", "EWY_Korea_ret_20d", "EWQ_France_zscore_60d", "LLY_zscore_60d", "EWC_Canada_zscore_60d", "Michigan_Sentiment_zscore_60d__zrel__AMZN_zscore_60d", "NFCI_ret_5d__div__SO_SouthernCo_ret_1d"], "is_new": false}, {"model_id": "v2_h5_CALM_RandomForest_N12", "algo": "RandomForest", "regime": "CALM", "horizon": 5, "n_features": 12, "F1_dir": 0.5143, "F1_UP_FORT": 0.2105, "F1_DOWN_FORT": 0.3433, "train_start": "2000-11-15", "sampler": "SMOTE", "best_params": "{}", "features": ["VIX_Price_zscore_60d__minus__HON_Honeywell_zscore_60d", "NFCI_ret_5d__div__NVDA_vol_20d", "vix_zscore_10d__ret5x__SO_SouthernCo_ret_1d", "VIX_Price_zscore_60d__div__VRP", "vix_zscore_10d__div__spx_abs_ret_max_5d", "EWY_Korea_ret_20d", "EWQ_France_zscore_60d", "LLY_zscore_60d", "EWC_Canada_zscore_60d", "Michigan_Sentiment_zscore_60d__zrel__AMZN_zscore_60d", "NFCI_ret_5d__div__SO_SouthernCo_ret_1d", "XLY_Disc_vol_20d"], "is_new": false}, {"model_id": "v2_h5_CALM_RandomForest_N13", "algo": "RandomForest", "regime": "CALM", "horizon": 5, "n_features": 13, "F1_dir": 0.5176, "F1_UP_FORT": 0.1739, "F1_DOWN_FORT": 0.3741, "train_start": "2000-11-15", "sampler": "SMOTE", "best_params": "{}", "features": ["VIX_Price_zscore_60d__minus__HON_Honeywell_zscore_60d", "NFCI_ret_5d__div__NVDA_vol_20d", "vix_zscore_10d__ret5x__SO_SouthernCo_ret_1d", "VIX_Price_zscore_60d__div__VRP", "vix_zscore_10d__div__spx_abs_ret_max_5d", "EWY_Korea_ret_20d", "EWQ_France_zscore_60d", "LLY_zscore_60d", "EWC_Canada_zscore_60d", "Michigan_Sentiment_zscore_60d__zrel__AMZN_zscore_60d", "NFCI_ret_5d__div__SO_SouthernCo_ret_1d", "XLY_Disc_vol_20d", "ASML_ASML_ret_5d__div__spx_momentum_3d"], "is_new": false}, {"model_id": "v2_h5_CALM_RandomForest_N15", "algo": "RandomForest", "regime": "CALM", "horizon": 5, "n_features": 15, "F1_dir": 0.5342, "F1_UP_FORT": 0.2364, "F1_DOWN_FORT": 0.3485, "train_start": "2000-11-15", "sampler": "SMOTE", "best_params": "{}", "features": ["VIX_Price_zscore_60d__minus__HON_Honeywell_zscore_60d", "NFCI_ret_5d__div__NVDA_vol_20d", "vix_zscore_10d__ret5x__SO_SouthernCo_ret_1d", "VIX_Price_zscore_60d__div__VRP", "vix_zscore_10d__div__spx_abs_ret_max_5d", "EWY_Korea_ret_20d", "EWQ_France_zscore_60d", "LLY_zscore_60d", "EWC_Canada_zscore_60d", "Michigan_Sentiment_zscore_60d__zrel__AMZN_zscore_60d", "NFCI_ret_5d__div__SO_SouthernCo_ret_1d", "XLY_Disc_vol_20d", "ASML_ASML_ret_5d__div__spx_momentum_3d", "NFCI_ret_5d", "Nikkei_Japan_zscore_60d__prod__AMZN_zscore_60d"], "is_new": false}, {"model_id": "v2_h5_CALM_RandomForest_N16", "algo": "RandomForest", "regime": "CALM", "horizon": 5, "n_features": 16, "F1_dir": 0.5258, "F1_UP_FORT": 0.2018, "F1_DOWN_FORT": 0.3761, "train_start": "2000-11-15", "sampler": "SMOTE", "best_params": "{}", "features": ["VIX_Price_zscore_60d__minus__HON_Honeywell_zscore_60d", "NFCI_ret_5d__div__NVDA_vol_20d", "vix_zscore_10d__ret5x__SO_SouthernCo_ret_1d", "VIX_Price_zscore_60d__div__VRP", "vix_zscore_10d__div__spx_abs_ret_max_5d", "EWY_Korea_ret_20d", "EWQ_France_zscore_60d", "LLY_zscore_60d", "EWC_Canada_zscore_60d", "Michigan_Sentiment_zscore_60d__zrel__AMZN_zscore_60d", "NFCI_ret_5d__div__SO_SouthernCo_ret_1d", "XLY_Disc_vol_20d", "ASML_ASML_ret_5d__div__spx_momentum_3d", "NFCI_ret_5d", "Nikkei_Japan_zscore_60d__prod__AMZN_zscore_60d", "T10Y2Y_Spread_ret_5d"], "is_new": false}, {"model_id": "v2_h5_CALM_RandomForest_N17", "algo": "RandomForest", "regime": "CALM", "horizon": 5, "n_features": 17, "F1_dir": 0.5688, "F1_UP_FORT": 0.2857, "F1_DOWN_FORT": 0.386, "train_start": "2000-11-15", "sampler": "SMOTE", "best_params": "{}", "features": ["VIX_Price_zscore_60d__minus__HON_Honeywell_zscore_60d", "NFCI_ret_5d__div__NVDA_vol_20d", "vix_zscore_10d__ret5x__SO_SouthernCo_ret_1d", "VIX_Price_zscore_60d__div__VRP", "vix_zscore_10d__div__spx_abs_ret_max_5d", "EWY_Korea_ret_20d", "EWQ_France_zscore_60d", "LLY_zscore_60d", "EWC_Canada_zscore_60d", "Michigan_Sentiment_zscore_60d__zrel__AMZN_zscore_60d", "NFCI_ret_5d__div__SO_SouthernCo_ret_1d", "XLY_Disc_vol_20d", "ASML_ASML_ret_5d__div__spx_momentum_3d", "NFCI_ret_5d", "Nikkei_Japan_zscore_60d__prod__AMZN_zscore_60d", "T10Y2Y_Spread_ret_5d", "TM_Telephone_vol_20d"], "is_new": false}, {"model_id": "v2_h5_CALM_RandomForest_N18", "algo": "RandomForest", "regime": "CALM", "horizon": 5, "n_features": 18, "F1_dir": 0.556, "F1_UP_FORT": 0.2807, "F1_DOWN_FORT": 0.3729, "train_start": "2000-11-15", "sampler": "SMOTE", "best_params": "{}", "features": ["VIX_Price_zscore_60d__minus__HON_Honeywell_zscore_60d", "NFCI_ret_5d__div__NVDA_vol_20d", "vix_zscore_10d__ret5x__SO_SouthernCo_ret_1d", "VIX_Price_zscore_60d__div__VRP", "vix_zscore_10d__div__spx_abs_ret_max_5d", "EWY_Korea_ret_20d", "EWQ_France_zscore_60d", "LLY_zscore_60d", "EWC_Canada_zscore_60d", "Michigan_Sentiment_zscore_60d__zrel__AMZN_zscore_60d", "NFCI_ret_5d__div__SO_SouthernCo_ret_1d", "XLY_Disc_vol_20d", "ASML_ASML_ret_5d__div__spx_momentum_3d", "NFCI_ret_5d", "Nikkei_Japan_zscore_60d__prod__AMZN_zscore_60d", "T10Y2Y_Spread_ret_5d", "TM_Telephone_vol_20d", "vix_zscore_10d__minus__HON_Honeywell_zscore_60d"], "is_new": false}, {"model_id": "v2_h5_CALM_LogisticRegression_N6", "algo": "LogisticRegression", "regime": "CALM", "horizon": 5, "n_features": 6, "F1_dir": 0.5016, "F1_UP_FORT": 0.2222, "F1_DOWN_FORT": 0.3425, "train_start": "2000-11-15", "sampler": "SMOTE", "best_params": "{}", "features": ["VIX_Price_zscore_60d__minus__HON_Honeywell_zscore_60d", "NFCI_ret_5d__div__NVDA_vol_20d", "vix_zscore_10d__ret5x__SO_SouthernCo_ret_1d", "VIX_Price_zscore_60d__div__VRP", "vix_zscore_10d__div__spx_abs_ret_max_5d", "EWY_Korea_ret_20d"], "is_new": false}, {"model_id": "v2_h5_CALM_LightGBM_Optuna_N9", "algo": "LightGBM", "regime": "CALM", "horizon": 5, "n_features": 9, "F1_dir": 0.5571, "F1_UP_FORT": 0.1964, "F1_DOWN_FORT": 0.3964, "train_start": "2000-11-15", "sampler": "SMOTE", "best_params": "{}", "features": ["VIX_Price_zscore_60d__minus__HON_Honeywell_zscore_60d", "NFCI_ret_5d__div__NVDA_vol_20d", "vix_zscore_10d__ret5x__SO_SouthernCo_ret_1d", "VIX_Price_zscore_60d__div__VRP", "vix_zscore_10d__div__spx_abs_ret_max_5d", "EWY_Korea_ret_20d", "EWQ_France_zscore_60d", "LLY_zscore_60d", "EWC_Canada_zscore_60d"], "is_new": false}, {"model_id": "v2_h5_CALM_LightGBM_OptunaCal_N9", "algo": "LightGBMCal", "regime": "CALM", "horizon": 5, "n_features": 9, "F1_dir": 0.5611, "F1_UP_FORT": 0.1964, "F1_DOWN_FORT": 0.4, "train_start": "2000-11-15", "sampler": "SMOTE", "best_params": "{}", "features": ["VIX_Price_zscore_60d__minus__HON_Honeywell_zscore_60d", "NFCI_ret_5d__div__NVDA_vol_20d", "vix_zscore_10d__ret5x__SO_SouthernCo_ret_1d", "VIX_Price_zscore_60d__div__VRP", "vix_zscore_10d__div__spx_abs_ret_max_5d", "EWY_Korea_ret_20d", "EWQ_France_zscore_60d", "LLY_zscore_60d", "EWC_Canada_zscore_60d"], "is_new": false}, {"model_id": "v2_h5_NORMAL_XGBoost_N5", "algo": "XGBoost", "regime": "NORMAL", "horizon": 5, "n_features": 5, "F1_dir": 0.5007, "F1_UP_FORT": 0.34, "F1_DOWN_FORT": 0.3119, "train_start": "2001-02-06", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VIX_Price_zscore_60d__prod__CCI_CrownCastle_vol_20d", "VIX_Price_zscore_60d__div__heston_theta", "VIX_Price_zscore_60d__div__hmm_p_stress", "EWY_Korea_ret_20d", "M_Macys_vol_20d"], "is_new": false}, {"model_id": "v2_h5_NORMAL_XGBoost_N6", "algo": "XGBoost", "regime": "NORMAL", "horizon": 5, "n_features": 6, "F1_dir": 0.5061, "F1_UP_FORT": 0.2783, "F1_DOWN_FORT": 0.3293, "train_start": "2001-02-06", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VIX_Price_zscore_60d__prod__CCI_CrownCastle_vol_20d", "VIX_Price_zscore_60d__div__heston_theta", "VIX_Price_zscore_60d__div__hmm_p_stress", "EWY_Korea_ret_20d", "M_Macys_vol_20d", "Core_CPI_zscore_60d__macross__VRP"], "is_new": false}, {"model_id": "v2_h5_NORMAL_XGBoost_N7", "algo": "XGBoost", "regime": "NORMAL", "horizon": 5, "n_features": 7, "F1_dir": 0.5138, "F1_UP_FORT": 0.33, "F1_DOWN_FORT": 0.343, "train_start": "2001-02-06", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VIX_Price_zscore_60d__prod__CCI_CrownCastle_vol_20d", "VIX_Price_zscore_60d__div__heston_theta", "VIX_Price_zscore_60d__div__hmm_p_stress", "EWY_Korea_ret_20d", "M_Macys_vol_20d", "Core_CPI_zscore_60d__macross__VRP", "VIX_Price_zscore_60d__div__vix_max_abs_ret_5d"], "is_new": false}, {"model_id": "v2_h5_NORMAL_XGBoost_N8", "algo": "XGBoost", "regime": "NORMAL", "horizon": 5, "n_features": 8, "F1_dir": 0.5451, "F1_UP_FORT": 0.2797, "F1_DOWN_FORT": 0.4116, "train_start": "2001-02-06", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VIX_Price_zscore_60d__prod__CCI_CrownCastle_vol_20d", "VIX_Price_zscore_60d__div__heston_theta", "VIX_Price_zscore_60d__div__hmm_p_stress", "EWY_Korea_ret_20d", "M_Macys_vol_20d", "Core_CPI_zscore_60d__macross__VRP", "VIX_Price_zscore_60d__div__vix_max_abs_ret_5d", "NFCI_ret_5d"], "is_new": false}, {"model_id": "v2_h5_NORMAL_XGBoost_N9", "algo": "XGBoost", "regime": "NORMAL", "horizon": 5, "n_features": 9, "F1_dir": 0.5289, "F1_UP_FORT": 0.2757, "F1_DOWN_FORT": 0.4258, "train_start": "2001-02-06", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VIX_Price_zscore_60d__prod__CCI_CrownCastle_vol_20d", "VIX_Price_zscore_60d__div__heston_theta", "VIX_Price_zscore_60d__div__hmm_p_stress", "EWY_Korea_ret_20d", "M_Macys_vol_20d", "Core_CPI_zscore_60d__macross__VRP", "VIX_Price_zscore_60d__div__vix_max_abs_ret_5d", "NFCI_ret_5d", "vix_zscore_10d__div__vix_vol_of_vol_10d"], "is_new": false}, {"model_id": "v2_h5_NORMAL_XGBoost_N10", "algo": "XGBoost", "regime": "NORMAL", "horizon": 5, "n_features": 10, "F1_dir": 0.5164, "F1_UP_FORT": 0.2588, "F1_DOWN_FORT": 0.4208, "train_start": "2001-02-06", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VIX_Price_zscore_60d__prod__CCI_CrownCastle_vol_20d", "VIX_Price_zscore_60d__div__heston_theta", "VIX_Price_zscore_60d__div__hmm_p_stress", "EWY_Korea_ret_20d", "M_Macys_vol_20d", "Core_CPI_zscore_60d__macross__VRP", "VIX_Price_zscore_60d__div__vix_max_abs_ret_5d", "NFCI_ret_5d", "vix_zscore_10d__div__vix_vol_of_vol_10d", "Nikkei_Japan_vol_20d"], "is_new": false}, {"model_id": "v2_h5_NORMAL_XGBoost_N16", "algo": "XGBoost", "regime": "NORMAL", "horizon": 5, "n_features": 16, "F1_dir": 0.5098, "F1_UP_FORT": 0.2157, "F1_DOWN_FORT": 0.439, "train_start": "2001-02-06", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VIX_Price_zscore_60d__prod__CCI_CrownCastle_vol_20d", "VIX_Price_zscore_60d__div__heston_theta", "VIX_Price_zscore_60d__div__hmm_p_stress", "EWY_Korea_ret_20d", "M_Macys_vol_20d", "Core_CPI_zscore_60d__macross__VRP", "VIX_Price_zscore_60d__div__vix_max_abs_ret_5d", "NFCI_ret_5d", "vix_zscore_10d__div__vix_vol_of_vol_10d", "Nikkei_Japan_vol_20d", "AMT_AmericanTower_zscore_60d__minus__vix_level", "3M_vol_20d", "VIX_Price_zscore_60d__div__VRP", "VIX_Price_zscore_60d__div__MCD_ret_5d", "SO_SouthernCo_ret_5d", "vix_vol_of_vol_10d__prod__heston_theta"], "is_new": false}, {"model_id": "v2_h5_NORMAL_XGBoost_N17", "algo": "XGBoost", "regime": "NORMAL", "horizon": 5, "n_features": 17, "F1_dir": 0.5185, "F1_UP_FORT": 0.2459, "F1_DOWN_FORT": 0.4467, "train_start": "2001-02-06", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VIX_Price_zscore_60d__prod__CCI_CrownCastle_vol_20d", "VIX_Price_zscore_60d__div__heston_theta", "VIX_Price_zscore_60d__div__hmm_p_stress", "EWY_Korea_ret_20d", "M_Macys_vol_20d", "Core_CPI_zscore_60d__macross__VRP", "VIX_Price_zscore_60d__div__vix_max_abs_ret_5d", "NFCI_ret_5d", "vix_zscore_10d__div__vix_vol_of_vol_10d", "Nikkei_Japan_vol_20d", "AMT_AmericanTower_zscore_60d__minus__vix_level", "3M_vol_20d", "VIX_Price_zscore_60d__div__VRP", "VIX_Price_zscore_60d__div__MCD_ret_5d", "SO_SouthernCo_ret_5d", "vix_vol_of_vol_10d__prod__heston_theta", "heston_var_ev_h1__minus__MCD_ret_5d"], "is_new": false}, {"model_id": "v2_h5_NORMAL_XGBoost_N18", "algo": "XGBoost", "regime": "NORMAL", "horizon": 5, "n_features": 18, "F1_dir": 0.5229, "F1_UP_FORT": 0.2324, "F1_DOWN_FORT": 0.4282, "train_start": "2001-02-06", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VIX_Price_zscore_60d__prod__CCI_CrownCastle_vol_20d", "VIX_Price_zscore_60d__div__heston_theta", "VIX_Price_zscore_60d__div__hmm_p_stress", "EWY_Korea_ret_20d", "M_Macys_vol_20d", "Core_CPI_zscore_60d__macross__VRP", "VIX_Price_zscore_60d__div__vix_max_abs_ret_5d", "NFCI_ret_5d", "vix_zscore_10d__div__vix_vol_of_vol_10d", "Nikkei_Japan_vol_20d", "AMT_AmericanTower_zscore_60d__minus__vix_level", "3M_vol_20d", "VIX_Price_zscore_60d__div__VRP", "VIX_Price_zscore_60d__div__MCD_ret_5d", "SO_SouthernCo_ret_5d", "vix_vol_of_vol_10d__prod__heston_theta", "heston_var_ev_h1__minus__MCD_ret_5d", "EMR_Emerson_ret_20d"], "is_new": false}, {"model_id": "v2_h5_NORMAL_XGBoost_N19", "algo": "XGBoost", "regime": "NORMAL", "horizon": 5, "n_features": 19, "F1_dir": 0.5299, "F1_UP_FORT": 0.2147, "F1_DOWN_FORT": 0.4444, "train_start": "2001-02-06", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VIX_Price_zscore_60d__prod__CCI_CrownCastle_vol_20d", "VIX_Price_zscore_60d__div__heston_theta", "VIX_Price_zscore_60d__div__hmm_p_stress", "EWY_Korea_ret_20d", "M_Macys_vol_20d", "Core_CPI_zscore_60d__macross__VRP", "VIX_Price_zscore_60d__div__vix_max_abs_ret_5d", "NFCI_ret_5d", "vix_zscore_10d__div__vix_vol_of_vol_10d", "Nikkei_Japan_vol_20d", "AMT_AmericanTower_zscore_60d__minus__vix_level", "3M_vol_20d", "VIX_Price_zscore_60d__div__VRP", "VIX_Price_zscore_60d__div__MCD_ret_5d", "SO_SouthernCo_ret_5d", "vix_vol_of_vol_10d__prod__heston_theta", "heston_var_ev_h1__minus__MCD_ret_5d", "EMR_Emerson_ret_20d", "vix_max_abs_ret_5d__macross__heston_theta"], "is_new": false}, {"model_id": "v2_h5_NORMAL_LightGBM_N5", "algo": "LightGBM", "regime": "NORMAL", "horizon": 5, "n_features": 5, "F1_dir": 0.5111, "F1_UP_FORT": 0.3644, "F1_DOWN_FORT": 0.3003, "train_start": "2001-02-06", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VIX_Price_zscore_60d__prod__CCI_CrownCastle_vol_20d", "VIX_Price_zscore_60d__div__heston_theta", "VIX_Price_zscore_60d__div__hmm_p_stress", "EWY_Korea_ret_20d", "M_Macys_vol_20d"], "is_new": false}, {"model_id": "v2_h5_NORMAL_LightGBM_N6", "algo": "LightGBM", "regime": "NORMAL", "horizon": 5, "n_features": 6, "F1_dir": 0.5032, "F1_UP_FORT": 0.2851, "F1_DOWN_FORT": 0.3427, "train_start": "2001-02-06", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VIX_Price_zscore_60d__prod__CCI_CrownCastle_vol_20d", "VIX_Price_zscore_60d__div__heston_theta", "VIX_Price_zscore_60d__div__hmm_p_stress", "EWY_Korea_ret_20d", "M_Macys_vol_20d", "Core_CPI_zscore_60d__macross__VRP"], "is_new": false}, {"model_id": "v2_h5_NORMAL_LightGBM_N8", "algo": "LightGBM", "regime": "NORMAL", "horizon": 5, "n_features": 8, "F1_dir": 0.5253, "F1_UP_FORT": 0.2768, "F1_DOWN_FORT": 0.4068, "train_start": "2001-02-06", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VIX_Price_zscore_60d__prod__CCI_CrownCastle_vol_20d", "VIX_Price_zscore_60d__div__heston_theta", "VIX_Price_zscore_60d__div__hmm_p_stress", "EWY_Korea_ret_20d", "M_Macys_vol_20d", "Core_CPI_zscore_60d__macross__VRP", "VIX_Price_zscore_60d__div__vix_max_abs_ret_5d", "NFCI_ret_5d"], "is_new": false}, {"model_id": "v2_h5_NORMAL_LightGBM_N9", "algo": "LightGBM", "regime": "NORMAL", "horizon": 5, "n_features": 9, "F1_dir": 0.5289, "F1_UP_FORT": 0.2835, "F1_DOWN_FORT": 0.4149, "train_start": "2001-02-06", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VIX_Price_zscore_60d__prod__CCI_CrownCastle_vol_20d", "VIX_Price_zscore_60d__div__heston_theta", "VIX_Price_zscore_60d__div__hmm_p_stress", "EWY_Korea_ret_20d", "M_Macys_vol_20d", "Core_CPI_zscore_60d__macross__VRP", "VIX_Price_zscore_60d__div__vix_max_abs_ret_5d", "NFCI_ret_5d", "vix_zscore_10d__div__vix_vol_of_vol_10d"], "is_new": false}, {"model_id": "v2_h5_NORMAL_LightGBM_N10", "algo": "LightGBM", "regime": "NORMAL", "horizon": 5, "n_features": 10, "F1_dir": 0.5192, "F1_UP_FORT": 0.248, "F1_DOWN_FORT": 0.3855, "train_start": "2001-02-06", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VIX_Price_zscore_60d__prod__CCI_CrownCastle_vol_20d", "VIX_Price_zscore_60d__div__heston_theta", "VIX_Price_zscore_60d__div__hmm_p_stress", "EWY_Korea_ret_20d", "M_Macys_vol_20d", "Core_CPI_zscore_60d__macross__VRP", "VIX_Price_zscore_60d__div__vix_max_abs_ret_5d", "NFCI_ret_5d", "vix_zscore_10d__div__vix_vol_of_vol_10d", "Nikkei_Japan_vol_20d"], "is_new": false}, {"model_id": "v2_h5_NORMAL_LightGBM_N12", "algo": "LightGBM", "regime": "NORMAL", "horizon": 5, "n_features": 12, "F1_dir": 0.5075, "F1_UP_FORT": 0.2209, "F1_DOWN_FORT": 0.4181, "train_start": "2001-02-06", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VIX_Price_zscore_60d__prod__CCI_CrownCastle_vol_20d", "VIX_Price_zscore_60d__div__heston_theta", "VIX_Price_zscore_60d__div__hmm_p_stress", "EWY_Korea_ret_20d", "M_Macys_vol_20d", "Core_CPI_zscore_60d__macross__VRP", "VIX_Price_zscore_60d__div__vix_max_abs_ret_5d", "NFCI_ret_5d", "vix_zscore_10d__div__vix_vol_of_vol_10d", "Nikkei_Japan_vol_20d", "AMT_AmericanTower_zscore_60d__minus__vix_level", "3M_vol_20d"], "is_new": false}, {"model_id": "v2_h5_NORMAL_LightGBM_N13", "algo": "LightGBM", "regime": "NORMAL", "horizon": 5, "n_features": 13, "F1_dir": 0.5121, "F1_UP_FORT": 0.226, "F1_DOWN_FORT": 0.4396, "train_start": "2001-02-06", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VIX_Price_zscore_60d__prod__CCI_CrownCastle_vol_20d", "VIX_Price_zscore_60d__div__heston_theta", "VIX_Price_zscore_60d__div__hmm_p_stress", "EWY_Korea_ret_20d", "M_Macys_vol_20d", "Core_CPI_zscore_60d__macross__VRP", "VIX_Price_zscore_60d__div__vix_max_abs_ret_5d", "NFCI_ret_5d", "vix_zscore_10d__div__vix_vol_of_vol_10d", "Nikkei_Japan_vol_20d", "AMT_AmericanTower_zscore_60d__minus__vix_level", "3M_vol_20d", "VIX_Price_zscore_60d__div__VRP"], "is_new": false}, {"model_id": "v2_h5_NORMAL_LightGBM_N16", "algo": "LightGBM", "regime": "NORMAL", "horizon": 5, "n_features": 16, "F1_dir": 0.5232, "F1_UP_FORT": 0.2254, "F1_DOWN_FORT": 0.4115, "train_start": "2001-02-06", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VIX_Price_zscore_60d__prod__CCI_CrownCastle_vol_20d", "VIX_Price_zscore_60d__div__heston_theta", "VIX_Price_zscore_60d__div__hmm_p_stress", "EWY_Korea_ret_20d", "M_Macys_vol_20d", "Core_CPI_zscore_60d__macross__VRP", "VIX_Price_zscore_60d__div__vix_max_abs_ret_5d", "NFCI_ret_5d", "vix_zscore_10d__div__vix_vol_of_vol_10d", "Nikkei_Japan_vol_20d", "AMT_AmericanTower_zscore_60d__minus__vix_level", "3M_vol_20d", "VIX_Price_zscore_60d__div__VRP", "VIX_Price_zscore_60d__div__MCD_ret_5d", "SO_SouthernCo_ret_5d", "vix_vol_of_vol_10d__prod__heston_theta"], "is_new": false}, {"model_id": "v2_h5_NORMAL_LightGBM_N17", "algo": "LightGBM", "regime": "NORMAL", "horizon": 5, "n_features": 17, "F1_dir": 0.5192, "F1_UP_FORT": 0.2597, "F1_DOWN_FORT": 0.4145, "train_start": "2001-02-06", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VIX_Price_zscore_60d__prod__CCI_CrownCastle_vol_20d", "VIX_Price_zscore_60d__div__heston_theta", "VIX_Price_zscore_60d__div__hmm_p_stress", "EWY_Korea_ret_20d", "M_Macys_vol_20d", "Core_CPI_zscore_60d__macross__VRP", "VIX_Price_zscore_60d__div__vix_max_abs_ret_5d", "NFCI_ret_5d", "vix_zscore_10d__div__vix_vol_of_vol_10d", "Nikkei_Japan_vol_20d", "AMT_AmericanTower_zscore_60d__minus__vix_level", "3M_vol_20d", "VIX_Price_zscore_60d__div__VRP", "VIX_Price_zscore_60d__div__MCD_ret_5d", "SO_SouthernCo_ret_5d", "vix_vol_of_vol_10d__prod__heston_theta", "heston_var_ev_h1__minus__MCD_ret_5d"], "is_new": false}, {"model_id": "v2_h5_NORMAL_LightGBM_N18", "algo": "LightGBM", "regime": "NORMAL", "horizon": 5, "n_features": 18, "F1_dir": 0.5276, "F1_UP_FORT": 0.2611, "F1_DOWN_FORT": 0.4289, "train_start": "2001-02-06", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VIX_Price_zscore_60d__prod__CCI_CrownCastle_vol_20d", "VIX_Price_zscore_60d__div__heston_theta", "VIX_Price_zscore_60d__div__hmm_p_stress", "EWY_Korea_ret_20d", "M_Macys_vol_20d", "Core_CPI_zscore_60d__macross__VRP", "VIX_Price_zscore_60d__div__vix_max_abs_ret_5d", "NFCI_ret_5d", "vix_zscore_10d__div__vix_vol_of_vol_10d", "Nikkei_Japan_vol_20d", "AMT_AmericanTower_zscore_60d__minus__vix_level", "3M_vol_20d", "VIX_Price_zscore_60d__div__VRP", "VIX_Price_zscore_60d__div__MCD_ret_5d", "SO_SouthernCo_ret_5d", "vix_vol_of_vol_10d__prod__heston_theta", "heston_var_ev_h1__minus__MCD_ret_5d", "EMR_Emerson_ret_20d"], "is_new": false}, {"model_id": "v2_h5_NORMAL_LightGBM_N19", "algo": "LightGBM", "regime": "NORMAL", "horizon": 5, "n_features": 19, "F1_dir": 0.5143, "F1_UP_FORT": 0.2305, "F1_DOWN_FORT": 0.4526, "train_start": "2001-02-06", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VIX_Price_zscore_60d__prod__CCI_CrownCastle_vol_20d", "VIX_Price_zscore_60d__div__heston_theta", "VIX_Price_zscore_60d__div__hmm_p_stress", "EWY_Korea_ret_20d", "M_Macys_vol_20d", "Core_CPI_zscore_60d__macross__VRP", "VIX_Price_zscore_60d__div__vix_max_abs_ret_5d", "NFCI_ret_5d", "vix_zscore_10d__div__vix_vol_of_vol_10d", "Nikkei_Japan_vol_20d", "AMT_AmericanTower_zscore_60d__minus__vix_level", "3M_vol_20d", "VIX_Price_zscore_60d__div__VRP", "VIX_Price_zscore_60d__div__MCD_ret_5d", "SO_SouthernCo_ret_5d", "vix_vol_of_vol_10d__prod__heston_theta", "heston_var_ev_h1__minus__MCD_ret_5d", "EMR_Emerson_ret_20d", "vix_max_abs_ret_5d__macross__heston_theta"], "is_new": false}, {"model_id": "v2_h5_NORMAL_GradientBoosting_N6", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 5, "n_features": 6, "F1_dir": 0.5226, "F1_UP_FORT": 0.2991, "F1_DOWN_FORT": 0.2701, "train_start": "2001-02-06", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VIX_Price_zscore_60d__prod__CCI_CrownCastle_vol_20d", "VIX_Price_zscore_60d__div__heston_theta", "VIX_Price_zscore_60d__div__hmm_p_stress", "EWY_Korea_ret_20d", "M_Macys_vol_20d", "Core_CPI_zscore_60d__macross__VRP"], "is_new": false}, {"model_id": "v2_h5_NORMAL_GradientBoosting_N7", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 5, "n_features": 7, "F1_dir": 0.5224, "F1_UP_FORT": 0.289, "F1_DOWN_FORT": 0.2997, "train_start": "2001-02-06", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VIX_Price_zscore_60d__prod__CCI_CrownCastle_vol_20d", "VIX_Price_zscore_60d__div__heston_theta", "VIX_Price_zscore_60d__div__hmm_p_stress", "EWY_Korea_ret_20d", "M_Macys_vol_20d", "Core_CPI_zscore_60d__macross__VRP", "VIX_Price_zscore_60d__div__vix_max_abs_ret_5d"], "is_new": false}, {"model_id": "v2_h5_NORMAL_GradientBoosting_N8", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 5, "n_features": 8, "F1_dir": 0.5188, "F1_UP_FORT": 0.2698, "F1_DOWN_FORT": 0.3552, "train_start": "2001-02-06", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VIX_Price_zscore_60d__prod__CCI_CrownCastle_vol_20d", "VIX_Price_zscore_60d__div__heston_theta", "VIX_Price_zscore_60d__div__hmm_p_stress", "EWY_Korea_ret_20d", "M_Macys_vol_20d", "Core_CPI_zscore_60d__macross__VRP", "VIX_Price_zscore_60d__div__vix_max_abs_ret_5d", "NFCI_ret_5d"], "is_new": false}, {"model_id": "v2_h5_NORMAL_GradientBoosting_N9", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 5, "n_features": 9, "F1_dir": 0.5133, "F1_UP_FORT": 0.2567, "F1_DOWN_FORT": 0.4203, "train_start": "2001-02-06", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VIX_Price_zscore_60d__prod__CCI_CrownCastle_vol_20d", "VIX_Price_zscore_60d__div__heston_theta", "VIX_Price_zscore_60d__div__hmm_p_stress", "EWY_Korea_ret_20d", "M_Macys_vol_20d", "Core_CPI_zscore_60d__macross__VRP", "VIX_Price_zscore_60d__div__vix_max_abs_ret_5d", "NFCI_ret_5d", "vix_zscore_10d__div__vix_vol_of_vol_10d"], "is_new": false}, {"model_id": "v2_h5_NORMAL_GradientBoosting_N10", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 5, "n_features": 10, "F1_dir": 0.512, "F1_UP_FORT": 0.234, "F1_DOWN_FORT": 0.4167, "train_start": "2001-02-06", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VIX_Price_zscore_60d__prod__CCI_CrownCastle_vol_20d", "VIX_Price_zscore_60d__div__heston_theta", "VIX_Price_zscore_60d__div__hmm_p_stress", "EWY_Korea_ret_20d", "M_Macys_vol_20d", "Core_CPI_zscore_60d__macross__VRP", "VIX_Price_zscore_60d__div__vix_max_abs_ret_5d", "NFCI_ret_5d", "vix_zscore_10d__div__vix_vol_of_vol_10d", "Nikkei_Japan_vol_20d"], "is_new": false}, {"model_id": "v2_h5_NORMAL_GradientBoosting_N11", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 5, "n_features": 11, "F1_dir": 0.5138, "F1_UP_FORT": 0.1905, "F1_DOWN_FORT": 0.4174, "train_start": "2001-02-06", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VIX_Price_zscore_60d__prod__CCI_CrownCastle_vol_20d", "VIX_Price_zscore_60d__div__heston_theta", "VIX_Price_zscore_60d__div__hmm_p_stress", "EWY_Korea_ret_20d", "M_Macys_vol_20d", "Core_CPI_zscore_60d__macross__VRP", "VIX_Price_zscore_60d__div__vix_max_abs_ret_5d", "NFCI_ret_5d", "vix_zscore_10d__div__vix_vol_of_vol_10d", "Nikkei_Japan_vol_20d", "AMT_AmericanTower_zscore_60d__minus__vix_level"], "is_new": false}, {"model_id": "v2_h5_NORMAL_GradientBoosting_N12", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 5, "n_features": 12, "F1_dir": 0.5075, "F1_UP_FORT": 0.2393, "F1_DOWN_FORT": 0.4428, "train_start": "2001-02-06", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VIX_Price_zscore_60d__prod__CCI_CrownCastle_vol_20d", "VIX_Price_zscore_60d__div__heston_theta", "VIX_Price_zscore_60d__div__hmm_p_stress", "EWY_Korea_ret_20d", "M_Macys_vol_20d", "Core_CPI_zscore_60d__macross__VRP", "VIX_Price_zscore_60d__div__vix_max_abs_ret_5d", "NFCI_ret_5d", "vix_zscore_10d__div__vix_vol_of_vol_10d", "Nikkei_Japan_vol_20d", "AMT_AmericanTower_zscore_60d__minus__vix_level", "3M_vol_20d"], "is_new": false}, {"model_id": "v2_h5_NORMAL_GradientBoosting_N13", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 5, "n_features": 13, "F1_dir": 0.5015, "F1_UP_FORT": 0.2273, "F1_DOWN_FORT": 0.4615, "train_start": "2001-02-06", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VIX_Price_zscore_60d__prod__CCI_CrownCastle_vol_20d", "VIX_Price_zscore_60d__div__heston_theta", "VIX_Price_zscore_60d__div__hmm_p_stress", "EWY_Korea_ret_20d", "M_Macys_vol_20d", "Core_CPI_zscore_60d__macross__VRP", "VIX_Price_zscore_60d__div__vix_max_abs_ret_5d", "NFCI_ret_5d", "vix_zscore_10d__div__vix_vol_of_vol_10d", "Nikkei_Japan_vol_20d", "AMT_AmericanTower_zscore_60d__minus__vix_level", "3M_vol_20d", "VIX_Price_zscore_60d__div__VRP"], "is_new": false}, {"model_id": "v2_h5_NORMAL_GradientBoosting_N15", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 5, "n_features": 15, "F1_dir": 0.5167, "F1_UP_FORT": 0.2543, "F1_DOWN_FORT": 0.4363, "train_start": "2001-02-06", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VIX_Price_zscore_60d__prod__CCI_CrownCastle_vol_20d", "VIX_Price_zscore_60d__div__heston_theta", "VIX_Price_zscore_60d__div__hmm_p_stress", "EWY_Korea_ret_20d", "M_Macys_vol_20d", "Core_CPI_zscore_60d__macross__VRP", "VIX_Price_zscore_60d__div__vix_max_abs_ret_5d", "NFCI_ret_5d", "vix_zscore_10d__div__vix_vol_of_vol_10d", "Nikkei_Japan_vol_20d", "AMT_AmericanTower_zscore_60d__minus__vix_level", "3M_vol_20d", "VIX_Price_zscore_60d__div__VRP", "VIX_Price_zscore_60d__div__MCD_ret_5d", "SO_SouthernCo_ret_5d"], "is_new": false}, {"model_id": "v2_h5_NORMAL_GradientBoosting_N16", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 5, "n_features": 16, "F1_dir": 0.537, "F1_UP_FORT": 0.2626, "F1_DOWN_FORT": 0.4329, "train_start": "2001-02-06", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VIX_Price_zscore_60d__prod__CCI_CrownCastle_vol_20d", "VIX_Price_zscore_60d__div__heston_theta", "VIX_Price_zscore_60d__div__hmm_p_stress", "EWY_Korea_ret_20d", "M_Macys_vol_20d", "Core_CPI_zscore_60d__macross__VRP", "VIX_Price_zscore_60d__div__vix_max_abs_ret_5d", "NFCI_ret_5d", "vix_zscore_10d__div__vix_vol_of_vol_10d", "Nikkei_Japan_vol_20d", "AMT_AmericanTower_zscore_60d__minus__vix_level", "3M_vol_20d", "VIX_Price_zscore_60d__div__VRP", "VIX_Price_zscore_60d__div__MCD_ret_5d", "SO_SouthernCo_ret_5d", "vix_vol_of_vol_10d__prod__heston_theta"], "is_new": false}, {"model_id": "v2_h5_NORMAL_GradientBoosting_N17", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 5, "n_features": 17, "F1_dir": 0.5133, "F1_UP_FORT": 0.2949, "F1_DOWN_FORT": 0.4232, "train_start": "2001-02-06", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VIX_Price_zscore_60d__prod__CCI_CrownCastle_vol_20d", "VIX_Price_zscore_60d__div__heston_theta", "VIX_Price_zscore_60d__div__hmm_p_stress", "EWY_Korea_ret_20d", "M_Macys_vol_20d", "Core_CPI_zscore_60d__macross__VRP", "VIX_Price_zscore_60d__div__vix_max_abs_ret_5d", "NFCI_ret_5d", "vix_zscore_10d__div__vix_vol_of_vol_10d", "Nikkei_Japan_vol_20d", "AMT_AmericanTower_zscore_60d__minus__vix_level", "3M_vol_20d", "VIX_Price_zscore_60d__div__VRP", "VIX_Price_zscore_60d__div__MCD_ret_5d", "SO_SouthernCo_ret_5d", "vix_vol_of_vol_10d__prod__heston_theta", "heston_var_ev_h1__minus__MCD_ret_5d"], "is_new": false}, {"model_id": "v2_h5_NORMAL_GradientBoosting_N18", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 5, "n_features": 18, "F1_dir": 0.5098, "F1_UP_FORT": 0.2575, "F1_DOWN_FORT": 0.459, "train_start": "2001-02-06", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VIX_Price_zscore_60d__prod__CCI_CrownCastle_vol_20d", "VIX_Price_zscore_60d__div__heston_theta", "VIX_Price_zscore_60d__div__hmm_p_stress", "EWY_Korea_ret_20d", "M_Macys_vol_20d", "Core_CPI_zscore_60d__macross__VRP", "VIX_Price_zscore_60d__div__vix_max_abs_ret_5d", "NFCI_ret_5d", "vix_zscore_10d__div__vix_vol_of_vol_10d", "Nikkei_Japan_vol_20d", "AMT_AmericanTower_zscore_60d__minus__vix_level", "3M_vol_20d", "VIX_Price_zscore_60d__div__VRP", "VIX_Price_zscore_60d__div__MCD_ret_5d", "SO_SouthernCo_ret_5d", "vix_vol_of_vol_10d__prod__heston_theta", "heston_var_ev_h1__minus__MCD_ret_5d", "EMR_Emerson_ret_20d"], "is_new": false}, {"model_id": "v2_h5_NORMAL_GradientBoosting_N19", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 5, "n_features": 19, "F1_dir": 0.5374, "F1_UP_FORT": 0.2466, "F1_DOWN_FORT": 0.4661, "train_start": "2001-02-06", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VIX_Price_zscore_60d__prod__CCI_CrownCastle_vol_20d", "VIX_Price_zscore_60d__div__heston_theta", "VIX_Price_zscore_60d__div__hmm_p_stress", "EWY_Korea_ret_20d", "M_Macys_vol_20d", "Core_CPI_zscore_60d__macross__VRP", "VIX_Price_zscore_60d__div__vix_max_abs_ret_5d", "NFCI_ret_5d", "vix_zscore_10d__div__vix_vol_of_vol_10d", "Nikkei_Japan_vol_20d", "AMT_AmericanTower_zscore_60d__minus__vix_level", "3M_vol_20d", "VIX_Price_zscore_60d__div__VRP", "VIX_Price_zscore_60d__div__MCD_ret_5d", "SO_SouthernCo_ret_5d", "vix_vol_of_vol_10d__prod__heston_theta", "heston_var_ev_h1__minus__MCD_ret_5d", "EMR_Emerson_ret_20d", "vix_max_abs_ret_5d__macross__heston_theta"], "is_new": false}, {"model_id": "v2_h5_NORMAL_RandomForest_N5", "algo": "RandomForest", "regime": "NORMAL", "horizon": 5, "n_features": 5, "F1_dir": 0.5044, "F1_UP_FORT": 0.3394, "F1_DOWN_FORT": 0.3641, "train_start": "2001-02-06", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VIX_Price_zscore_60d__prod__CCI_CrownCastle_vol_20d", "VIX_Price_zscore_60d__div__heston_theta", "VIX_Price_zscore_60d__div__hmm_p_stress", "EWY_Korea_ret_20d", "M_Macys_vol_20d"], "is_new": false}, {"model_id": "v2_h5_NORMAL_RandomForest_N6", "algo": "RandomForest", "regime": "NORMAL", "horizon": 5, "n_features": 6, "F1_dir": 0.5322, "F1_UP_FORT": 0.3106, "F1_DOWN_FORT": 0.3478, "train_start": "2001-02-06", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VIX_Price_zscore_60d__prod__CCI_CrownCastle_vol_20d", "VIX_Price_zscore_60d__div__heston_theta", "VIX_Price_zscore_60d__div__hmm_p_stress", "EWY_Korea_ret_20d", "M_Macys_vol_20d", "Core_CPI_zscore_60d__macross__VRP"], "is_new": false}, {"model_id": "v2_h5_NORMAL_RandomForest_N7", "algo": "RandomForest", "regime": "NORMAL", "horizon": 5, "n_features": 7, "F1_dir": 0.5267, "F1_UP_FORT": 0.3306, "F1_DOWN_FORT": 0.3711, "train_start": "2001-02-06", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VIX_Price_zscore_60d__prod__CCI_CrownCastle_vol_20d", "VIX_Price_zscore_60d__div__heston_theta", "VIX_Price_zscore_60d__div__hmm_p_stress", "EWY_Korea_ret_20d", "M_Macys_vol_20d", "Core_CPI_zscore_60d__macross__VRP", "VIX_Price_zscore_60d__div__vix_max_abs_ret_5d"], "is_new": false}, {"model_id": "v2_h5_NORMAL_RandomForest_N8", "algo": "RandomForest", "regime": "NORMAL", "horizon": 5, "n_features": 8, "F1_dir": 0.5635, "F1_UP_FORT": 0.2835, "F1_DOWN_FORT": 0.4449, "train_start": "2001-02-06", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VIX_Price_zscore_60d__prod__CCI_CrownCastle_vol_20d", "VIX_Price_zscore_60d__div__heston_theta", "VIX_Price_zscore_60d__div__hmm_p_stress", "EWY_Korea_ret_20d", "M_Macys_vol_20d", "Core_CPI_zscore_60d__macross__VRP", "VIX_Price_zscore_60d__div__vix_max_abs_ret_5d", "NFCI_ret_5d"], "is_new": false}, {"model_id": "v2_h5_NORMAL_RandomForest_N9", "algo": "RandomForest", "regime": "NORMAL", "horizon": 5, "n_features": 9, "F1_dir": 0.5547, "F1_UP_FORT": 0.272, "F1_DOWN_FORT": 0.4458, "train_start": "2001-02-06", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VIX_Price_zscore_60d__prod__CCI_CrownCastle_vol_20d", "VIX_Price_zscore_60d__div__heston_theta", "VIX_Price_zscore_60d__div__hmm_p_stress", "EWY_Korea_ret_20d", "M_Macys_vol_20d", "Core_CPI_zscore_60d__macross__VRP", "VIX_Price_zscore_60d__div__vix_max_abs_ret_5d", "NFCI_ret_5d", "vix_zscore_10d__div__vix_vol_of_vol_10d"], "is_new": false}, {"model_id": "v2_h5_NORMAL_RandomForest_N10", "algo": "RandomForest", "regime": "NORMAL", "horizon": 5, "n_features": 10, "F1_dir": 0.5594, "F1_UP_FORT": 0.2644, "F1_DOWN_FORT": 0.4575, "train_start": "2001-02-06", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VIX_Price_zscore_60d__prod__CCI_CrownCastle_vol_20d", "VIX_Price_zscore_60d__div__heston_theta", "VIX_Price_zscore_60d__div__hmm_p_stress", "EWY_Korea_ret_20d", "M_Macys_vol_20d", "Core_CPI_zscore_60d__macross__VRP", "VIX_Price_zscore_60d__div__vix_max_abs_ret_5d", "NFCI_ret_5d", "vix_zscore_10d__div__vix_vol_of_vol_10d", "Nikkei_Japan_vol_20d"], "is_new": false}, {"model_id": "v2_h5_NORMAL_RandomForest_N11", "algo": "RandomForest", "regime": "NORMAL", "horizon": 5, "n_features": 11, "F1_dir": 0.5539, "F1_UP_FORT": 0.2388, "F1_DOWN_FORT": 0.4689, "train_start": "2001-02-06", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VIX_Price_zscore_60d__prod__CCI_CrownCastle_vol_20d", "VIX_Price_zscore_60d__div__heston_theta", "VIX_Price_zscore_60d__div__hmm_p_stress", "EWY_Korea_ret_20d", "M_Macys_vol_20d", "Core_CPI_zscore_60d__macross__VRP", "VIX_Price_zscore_60d__div__vix_max_abs_ret_5d", "NFCI_ret_5d", "vix_zscore_10d__div__vix_vol_of_vol_10d", "Nikkei_Japan_vol_20d", "AMT_AmericanTower_zscore_60d__minus__vix_level"], "is_new": false}, {"model_id": "v2_h5_NORMAL_RandomForest_N12", "algo": "RandomForest", "regime": "NORMAL", "horizon": 5, "n_features": 12, "F1_dir": 0.5515, "F1_UP_FORT": 0.2655, "F1_DOWN_FORT": 0.4694, "train_start": "2001-02-06", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VIX_Price_zscore_60d__prod__CCI_CrownCastle_vol_20d", "VIX_Price_zscore_60d__div__heston_theta", "VIX_Price_zscore_60d__div__hmm_p_stress", "EWY_Korea_ret_20d", "M_Macys_vol_20d", "Core_CPI_zscore_60d__macross__VRP", "VIX_Price_zscore_60d__div__vix_max_abs_ret_5d", "NFCI_ret_5d", "vix_zscore_10d__div__vix_vol_of_vol_10d", "Nikkei_Japan_vol_20d", "AMT_AmericanTower_zscore_60d__minus__vix_level", "3M_vol_20d"], "is_new": false}, {"model_id": "v2_h5_NORMAL_RandomForest_N13", "algo": "RandomForest", "regime": "NORMAL", "horizon": 5, "n_features": 13, "F1_dir": 0.5403, "F1_UP_FORT": 0.2955, "F1_DOWN_FORT": 0.4664, "train_start": "2001-02-06", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VIX_Price_zscore_60d__prod__CCI_CrownCastle_vol_20d", "VIX_Price_zscore_60d__div__heston_theta", "VIX_Price_zscore_60d__div__hmm_p_stress", "EWY_Korea_ret_20d", "M_Macys_vol_20d", "Core_CPI_zscore_60d__macross__VRP", "VIX_Price_zscore_60d__div__vix_max_abs_ret_5d", "NFCI_ret_5d", "vix_zscore_10d__div__vix_vol_of_vol_10d", "Nikkei_Japan_vol_20d", "AMT_AmericanTower_zscore_60d__minus__vix_level", "3M_vol_20d", "VIX_Price_zscore_60d__div__VRP"], "is_new": false}, {"model_id": "v2_h5_NORMAL_RandomForest_N14", "algo": "RandomForest", "regime": "NORMAL", "horizon": 5, "n_features": 14, "F1_dir": 0.541, "F1_UP_FORT": 0.2651, "F1_DOWN_FORT": 0.4664, "train_start": "2001-02-06", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VIX_Price_zscore_60d__prod__CCI_CrownCastle_vol_20d", "VIX_Price_zscore_60d__div__heston_theta", "VIX_Price_zscore_60d__div__hmm_p_stress", "EWY_Korea_ret_20d", "M_Macys_vol_20d", "Core_CPI_zscore_60d__macross__VRP", "VIX_Price_zscore_60d__div__vix_max_abs_ret_5d", "NFCI_ret_5d", "vix_zscore_10d__div__vix_vol_of_vol_10d", "Nikkei_Japan_vol_20d", "AMT_AmericanTower_zscore_60d__minus__vix_level", "3M_vol_20d", "VIX_Price_zscore_60d__div__VRP", "VIX_Price_zscore_60d__div__MCD_ret_5d"], "is_new": false}, {"model_id": "v2_h5_NORMAL_RandomForest_N15", "algo": "RandomForest", "regime": "NORMAL", "horizon": 5, "n_features": 15, "F1_dir": 0.5518, "F1_UP_FORT": 0.3236, "F1_DOWN_FORT": 0.4686, "train_start": "2001-02-06", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VIX_Price_zscore_60d__prod__CCI_CrownCastle_vol_20d", "VIX_Price_zscore_60d__div__heston_theta", "VIX_Price_zscore_60d__div__hmm_p_stress", "EWY_Korea_ret_20d", "M_Macys_vol_20d", "Core_CPI_zscore_60d__macross__VRP", "VIX_Price_zscore_60d__div__vix_max_abs_ret_5d", "NFCI_ret_5d", "vix_zscore_10d__div__vix_vol_of_vol_10d", "Nikkei_Japan_vol_20d", "AMT_AmericanTower_zscore_60d__minus__vix_level", "3M_vol_20d", "VIX_Price_zscore_60d__div__VRP", "VIX_Price_zscore_60d__div__MCD_ret_5d", "SO_SouthernCo_ret_5d"], "is_new": false}, {"model_id": "v2_h5_NORMAL_RandomForest_N16", "algo": "RandomForest", "regime": "NORMAL", "horizon": 5, "n_features": 16, "F1_dir": 0.5559, "F1_UP_FORT": 0.2772, "F1_DOWN_FORT": 0.4589, "train_start": "2001-02-06", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VIX_Price_zscore_60d__prod__CCI_CrownCastle_vol_20d", "VIX_Price_zscore_60d__div__heston_theta", "VIX_Price_zscore_60d__div__hmm_p_stress", "EWY_Korea_ret_20d", "M_Macys_vol_20d", "Core_CPI_zscore_60d__macross__VRP", "VIX_Price_zscore_60d__div__vix_max_abs_ret_5d", "NFCI_ret_5d", "vix_zscore_10d__div__vix_vol_of_vol_10d", "Nikkei_Japan_vol_20d", "AMT_AmericanTower_zscore_60d__minus__vix_level", "3M_vol_20d", "VIX_Price_zscore_60d__div__VRP", "VIX_Price_zscore_60d__div__MCD_ret_5d", "SO_SouthernCo_ret_5d", "vix_vol_of_vol_10d__prod__heston_theta"], "is_new": false}, {"model_id": "v2_h5_NORMAL_RandomForest_N17", "algo": "RandomForest", "regime": "NORMAL", "horizon": 5, "n_features": 17, "F1_dir": 0.5502, "F1_UP_FORT": 0.2873, "F1_DOWN_FORT": 0.4686, "train_start": "2001-02-06", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VIX_Price_zscore_60d__prod__CCI_CrownCastle_vol_20d", "VIX_Price_zscore_60d__div__heston_theta", "VIX_Price_zscore_60d__div__hmm_p_stress", "EWY_Korea_ret_20d", "M_Macys_vol_20d", "Core_CPI_zscore_60d__macross__VRP", "VIX_Price_zscore_60d__div__vix_max_abs_ret_5d", "NFCI_ret_5d", "vix_zscore_10d__div__vix_vol_of_vol_10d", "Nikkei_Japan_vol_20d", "AMT_AmericanTower_zscore_60d__minus__vix_level", "3M_vol_20d", "VIX_Price_zscore_60d__div__VRP", "VIX_Price_zscore_60d__div__MCD_ret_5d", "SO_SouthernCo_ret_5d", "vix_vol_of_vol_10d__prod__heston_theta", "heston_var_ev_h1__minus__MCD_ret_5d"], "is_new": false}, {"model_id": "v2_h5_NORMAL_RandomForest_N18", "algo": "RandomForest", "regime": "NORMAL", "horizon": 5, "n_features": 18, "F1_dir": 0.559, "F1_UP_FORT": 0.3133, "F1_DOWN_FORT": 0.4711, "train_start": "2001-02-06", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VIX_Price_zscore_60d__prod__CCI_CrownCastle_vol_20d", "VIX_Price_zscore_60d__div__heston_theta", "VIX_Price_zscore_60d__div__hmm_p_stress", "EWY_Korea_ret_20d", "M_Macys_vol_20d", "Core_CPI_zscore_60d__macross__VRP", "VIX_Price_zscore_60d__div__vix_max_abs_ret_5d", "NFCI_ret_5d", "vix_zscore_10d__div__vix_vol_of_vol_10d", "Nikkei_Japan_vol_20d", "AMT_AmericanTower_zscore_60d__minus__vix_level", "3M_vol_20d", "VIX_Price_zscore_60d__div__VRP", "VIX_Price_zscore_60d__div__MCD_ret_5d", "SO_SouthernCo_ret_5d", "vix_vol_of_vol_10d__prod__heston_theta", "heston_var_ev_h1__minus__MCD_ret_5d", "EMR_Emerson_ret_20d"], "is_new": false}, {"model_id": "v2_h5_NORMAL_RandomForest_N19", "algo": "RandomForest", "regime": "NORMAL", "horizon": 5, "n_features": 19, "F1_dir": 0.5534, "F1_UP_FORT": 0.2695, "F1_DOWN_FORT": 0.4564, "train_start": "2001-02-06", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VIX_Price_zscore_60d__prod__CCI_CrownCastle_vol_20d", "VIX_Price_zscore_60d__div__heston_theta", "VIX_Price_zscore_60d__div__hmm_p_stress", "EWY_Korea_ret_20d", "M_Macys_vol_20d", "Core_CPI_zscore_60d__macross__VRP", "VIX_Price_zscore_60d__div__vix_max_abs_ret_5d", "NFCI_ret_5d", "vix_zscore_10d__div__vix_vol_of_vol_10d", "Nikkei_Japan_vol_20d", "AMT_AmericanTower_zscore_60d__minus__vix_level", "3M_vol_20d", "VIX_Price_zscore_60d__div__VRP", "VIX_Price_zscore_60d__div__MCD_ret_5d", "SO_SouthernCo_ret_5d", "vix_vol_of_vol_10d__prod__heston_theta", "heston_var_ev_h1__minus__MCD_ret_5d", "EMR_Emerson_ret_20d", "vix_max_abs_ret_5d__macross__heston_theta"], "is_new": false}, {"model_id": "v2_h5_NORMAL_LogisticRegression_N5", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 5, "n_features": 5, "F1_dir": 0.5212, "F1_UP_FORT": 0.3556, "F1_DOWN_FORT": 0.3762, "train_start": "2001-02-06", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VIX_Price_zscore_60d__prod__CCI_CrownCastle_vol_20d", "VIX_Price_zscore_60d__div__heston_theta", "VIX_Price_zscore_60d__div__hmm_p_stress", "EWY_Korea_ret_20d", "M_Macys_vol_20d"], "is_new": false}, {"model_id": "v2_h5_NORMAL_LogisticRegression_N6", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 5, "n_features": 6, "F1_dir": 0.5391, "F1_UP_FORT": 0.3381, "F1_DOWN_FORT": 0.3758, "train_start": "2001-02-06", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VIX_Price_zscore_60d__prod__CCI_CrownCastle_vol_20d", "VIX_Price_zscore_60d__div__heston_theta", "VIX_Price_zscore_60d__div__hmm_p_stress", "EWY_Korea_ret_20d", "M_Macys_vol_20d", "Core_CPI_zscore_60d__macross__VRP"], "is_new": false}, {"model_id": "v2_h5_NORMAL_LogisticRegression_N7", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 5, "n_features": 7, "F1_dir": 0.5295, "F1_UP_FORT": 0.3192, "F1_DOWN_FORT": 0.3815, "train_start": "2001-02-06", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VIX_Price_zscore_60d__prod__CCI_CrownCastle_vol_20d", "VIX_Price_zscore_60d__div__heston_theta", "VIX_Price_zscore_60d__div__hmm_p_stress", "EWY_Korea_ret_20d", "M_Macys_vol_20d", "Core_CPI_zscore_60d__macross__VRP", "VIX_Price_zscore_60d__div__vix_max_abs_ret_5d"], "is_new": false}, {"model_id": "v2_h5_NORMAL_LogisticRegression_N8", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 5, "n_features": 8, "F1_dir": 0.5473, "F1_UP_FORT": 0.3014, "F1_DOWN_FORT": 0.4146, "train_start": "2001-02-06", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VIX_Price_zscore_60d__prod__CCI_CrownCastle_vol_20d", "VIX_Price_zscore_60d__div__heston_theta", "VIX_Price_zscore_60d__div__hmm_p_stress", "EWY_Korea_ret_20d", "M_Macys_vol_20d", "Core_CPI_zscore_60d__macross__VRP", "VIX_Price_zscore_60d__div__vix_max_abs_ret_5d", "NFCI_ret_5d"], "is_new": false}, {"model_id": "v2_h5_NORMAL_LogisticRegression_N9", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 5, "n_features": 9, "F1_dir": 0.5498, "F1_UP_FORT": 0.3095, "F1_DOWN_FORT": 0.4087, "train_start": "2001-02-06", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VIX_Price_zscore_60d__prod__CCI_CrownCastle_vol_20d", "VIX_Price_zscore_60d__div__heston_theta", "VIX_Price_zscore_60d__div__hmm_p_stress", "EWY_Korea_ret_20d", "M_Macys_vol_20d", "Core_CPI_zscore_60d__macross__VRP", "VIX_Price_zscore_60d__div__vix_max_abs_ret_5d", "NFCI_ret_5d", "vix_zscore_10d__div__vix_vol_of_vol_10d"], "is_new": false}, {"model_id": "v2_h5_NORMAL_LogisticRegression_N10", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 5, "n_features": 10, "F1_dir": 0.5324, "F1_UP_FORT": 0.3243, "F1_DOWN_FORT": 0.4077, "train_start": "2001-02-06", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VIX_Price_zscore_60d__prod__CCI_CrownCastle_vol_20d", "VIX_Price_zscore_60d__div__heston_theta", "VIX_Price_zscore_60d__div__hmm_p_stress", "EWY_Korea_ret_20d", "M_Macys_vol_20d", "Core_CPI_zscore_60d__macross__VRP", "VIX_Price_zscore_60d__div__vix_max_abs_ret_5d", "NFCI_ret_5d", "vix_zscore_10d__div__vix_vol_of_vol_10d", "Nikkei_Japan_vol_20d"], "is_new": false}, {"model_id": "v2_h5_NORMAL_LogisticRegression_N11", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 5, "n_features": 11, "F1_dir": 0.5375, "F1_UP_FORT": 0.3187, "F1_DOWN_FORT": 0.4426, "train_start": "2001-02-06", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VIX_Price_zscore_60d__prod__CCI_CrownCastle_vol_20d", "VIX_Price_zscore_60d__div__heston_theta", "VIX_Price_zscore_60d__div__hmm_p_stress", "EWY_Korea_ret_20d", "M_Macys_vol_20d", "Core_CPI_zscore_60d__macross__VRP", "VIX_Price_zscore_60d__div__vix_max_abs_ret_5d", "NFCI_ret_5d", "vix_zscore_10d__div__vix_vol_of_vol_10d", "Nikkei_Japan_vol_20d", "AMT_AmericanTower_zscore_60d__minus__vix_level"], "is_new": false}, {"model_id": "v2_h5_NORMAL_LogisticRegression_N12", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 5, "n_features": 12, "F1_dir": 0.5398, "F1_UP_FORT": 0.2877, "F1_DOWN_FORT": 0.4229, "train_start": "2001-02-06", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VIX_Price_zscore_60d__prod__CCI_CrownCastle_vol_20d", "VIX_Price_zscore_60d__div__heston_theta", "VIX_Price_zscore_60d__div__hmm_p_stress", "EWY_Korea_ret_20d", "M_Macys_vol_20d", "Core_CPI_zscore_60d__macross__VRP", "VIX_Price_zscore_60d__div__vix_max_abs_ret_5d", "NFCI_ret_5d", "vix_zscore_10d__div__vix_vol_of_vol_10d", "Nikkei_Japan_vol_20d", "AMT_AmericanTower_zscore_60d__minus__vix_level", "3M_vol_20d"], "is_new": false}, {"model_id": "v2_h5_NORMAL_LogisticRegression_N13", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 5, "n_features": 13, "F1_dir": 0.5398, "F1_UP_FORT": 0.2808, "F1_DOWN_FORT": 0.4306, "train_start": "2001-02-06", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VIX_Price_zscore_60d__prod__CCI_CrownCastle_vol_20d", "VIX_Price_zscore_60d__div__heston_theta", "VIX_Price_zscore_60d__div__hmm_p_stress", "EWY_Korea_ret_20d", "M_Macys_vol_20d", "Core_CPI_zscore_60d__macross__VRP", "VIX_Price_zscore_60d__div__vix_max_abs_ret_5d", "NFCI_ret_5d", "vix_zscore_10d__div__vix_vol_of_vol_10d", "Nikkei_Japan_vol_20d", "AMT_AmericanTower_zscore_60d__minus__vix_level", "3M_vol_20d", "VIX_Price_zscore_60d__div__VRP"], "is_new": false}, {"model_id": "v2_h5_NORMAL_LogisticRegression_N14", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 5, "n_features": 14, "F1_dir": 0.537, "F1_UP_FORT": 0.267, "F1_DOWN_FORT": 0.4298, "train_start": "2001-02-06", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VIX_Price_zscore_60d__prod__CCI_CrownCastle_vol_20d", "VIX_Price_zscore_60d__div__heston_theta", "VIX_Price_zscore_60d__div__hmm_p_stress", "EWY_Korea_ret_20d", "M_Macys_vol_20d", "Core_CPI_zscore_60d__macross__VRP", "VIX_Price_zscore_60d__div__vix_max_abs_ret_5d", "NFCI_ret_5d", "vix_zscore_10d__div__vix_vol_of_vol_10d", "Nikkei_Japan_vol_20d", "AMT_AmericanTower_zscore_60d__minus__vix_level", "3M_vol_20d", "VIX_Price_zscore_60d__div__VRP", "VIX_Price_zscore_60d__div__MCD_ret_5d"], "is_new": false}, {"model_id": "v2_h5_NORMAL_LogisticRegression_N15", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 5, "n_features": 15, "F1_dir": 0.5329, "F1_UP_FORT": 0.2843, "F1_DOWN_FORT": 0.4444, "train_start": "2001-02-06", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VIX_Price_zscore_60d__prod__CCI_CrownCastle_vol_20d", "VIX_Price_zscore_60d__div__heston_theta", "VIX_Price_zscore_60d__div__hmm_p_stress", "EWY_Korea_ret_20d", "M_Macys_vol_20d", "Core_CPI_zscore_60d__macross__VRP", "VIX_Price_zscore_60d__div__vix_max_abs_ret_5d", "NFCI_ret_5d", "vix_zscore_10d__div__vix_vol_of_vol_10d", "Nikkei_Japan_vol_20d", "AMT_AmericanTower_zscore_60d__minus__vix_level", "3M_vol_20d", "VIX_Price_zscore_60d__div__VRP", "VIX_Price_zscore_60d__div__MCD_ret_5d", "SO_SouthernCo_ret_5d"], "is_new": false}, {"model_id": "v2_h5_NORMAL_LogisticRegression_N16", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 5, "n_features": 16, "F1_dir": 0.5433, "F1_UP_FORT": 0.245, "F1_DOWN_FORT": 0.4642, "train_start": "2001-02-06", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VIX_Price_zscore_60d__prod__CCI_CrownCastle_vol_20d", "VIX_Price_zscore_60d__div__heston_theta", "VIX_Price_zscore_60d__div__hmm_p_stress", "EWY_Korea_ret_20d", "M_Macys_vol_20d", "Core_CPI_zscore_60d__macross__VRP", "VIX_Price_zscore_60d__div__vix_max_abs_ret_5d", "NFCI_ret_5d", "vix_zscore_10d__div__vix_vol_of_vol_10d", "Nikkei_Japan_vol_20d", "AMT_AmericanTower_zscore_60d__minus__vix_level", "3M_vol_20d", "VIX_Price_zscore_60d__div__VRP", "VIX_Price_zscore_60d__div__MCD_ret_5d", "SO_SouthernCo_ret_5d", "vix_vol_of_vol_10d__prod__heston_theta"], "is_new": false}, {"model_id": "v2_h5_NORMAL_LogisticRegression_N17", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 5, "n_features": 17, "F1_dir": 0.5433, "F1_UP_FORT": 0.2339, "F1_DOWN_FORT": 0.462, "train_start": "2001-02-06", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VIX_Price_zscore_60d__prod__CCI_CrownCastle_vol_20d", "VIX_Price_zscore_60d__div__heston_theta", "VIX_Price_zscore_60d__div__hmm_p_stress", "EWY_Korea_ret_20d", "M_Macys_vol_20d", "Core_CPI_zscore_60d__macross__VRP", "VIX_Price_zscore_60d__div__vix_max_abs_ret_5d", "NFCI_ret_5d", "vix_zscore_10d__div__vix_vol_of_vol_10d", "Nikkei_Japan_vol_20d", "AMT_AmericanTower_zscore_60d__minus__vix_level", "3M_vol_20d", "VIX_Price_zscore_60d__div__VRP", "VIX_Price_zscore_60d__div__MCD_ret_5d", "SO_SouthernCo_ret_5d", "vix_vol_of_vol_10d__prod__heston_theta", "heston_var_ev_h1__minus__MCD_ret_5d"], "is_new": false}, {"model_id": "v2_h5_NORMAL_LogisticRegression_N18", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 5, "n_features": 18, "F1_dir": 0.5466, "F1_UP_FORT": 0.2486, "F1_DOWN_FORT": 0.4558, "train_start": "2001-02-06", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VIX_Price_zscore_60d__prod__CCI_CrownCastle_vol_20d", "VIX_Price_zscore_60d__div__heston_theta", "VIX_Price_zscore_60d__div__hmm_p_stress", "EWY_Korea_ret_20d", "M_Macys_vol_20d", "Core_CPI_zscore_60d__macross__VRP", "VIX_Price_zscore_60d__div__vix_max_abs_ret_5d", "NFCI_ret_5d", "vix_zscore_10d__div__vix_vol_of_vol_10d", "Nikkei_Japan_vol_20d", "AMT_AmericanTower_zscore_60d__minus__vix_level", "3M_vol_20d", "VIX_Price_zscore_60d__div__VRP", "VIX_Price_zscore_60d__div__MCD_ret_5d", "SO_SouthernCo_ret_5d", "vix_vol_of_vol_10d__prod__heston_theta", "heston_var_ev_h1__minus__MCD_ret_5d", "EMR_Emerson_ret_20d"], "is_new": false}, {"model_id": "v2_h5_NORMAL_LogisticRegression_N19", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 5, "n_features": 19, "F1_dir": 0.5405, "F1_UP_FORT": 0.2471, "F1_DOWN_FORT": 0.4615, "train_start": "2001-02-06", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VIX_Price_zscore_60d__prod__CCI_CrownCastle_vol_20d", "VIX_Price_zscore_60d__div__heston_theta", "VIX_Price_zscore_60d__div__hmm_p_stress", "EWY_Korea_ret_20d", "M_Macys_vol_20d", "Core_CPI_zscore_60d__macross__VRP", "VIX_Price_zscore_60d__div__vix_max_abs_ret_5d", "NFCI_ret_5d", "vix_zscore_10d__div__vix_vol_of_vol_10d", "Nikkei_Japan_vol_20d", "AMT_AmericanTower_zscore_60d__minus__vix_level", "3M_vol_20d", "VIX_Price_zscore_60d__div__VRP", "VIX_Price_zscore_60d__div__MCD_ret_5d", "SO_SouthernCo_ret_5d", "vix_vol_of_vol_10d__prod__heston_theta", "heston_var_ev_h1__minus__MCD_ret_5d", "EMR_Emerson_ret_20d", "vix_max_abs_ret_5d__macross__heston_theta"], "is_new": false}, {"model_id": "v2_h5_NORMAL_GradientBoosting_Optuna_N19", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 5, "n_features": 19, "F1_dir": 0.5393, "F1_UP_FORT": 0.2674, "F1_DOWN_FORT": 0.4469, "train_start": "2001-02-06", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VIX_Price_zscore_60d__prod__CCI_CrownCastle_vol_20d", "VIX_Price_zscore_60d__div__heston_theta", "VIX_Price_zscore_60d__div__hmm_p_stress", "EWY_Korea_ret_20d", "M_Macys_vol_20d", "Core_CPI_zscore_60d__macross__VRP", "VIX_Price_zscore_60d__div__vix_max_abs_ret_5d", "NFCI_ret_5d", "vix_zscore_10d__div__vix_vol_of_vol_10d", "Nikkei_Japan_vol_20d", "AMT_AmericanTower_zscore_60d__minus__vix_level", "3M_vol_20d", "VIX_Price_zscore_60d__div__VRP", "VIX_Price_zscore_60d__div__MCD_ret_5d", "SO_SouthernCo_ret_5d", "vix_vol_of_vol_10d__prod__heston_theta", "heston_var_ev_h1__minus__MCD_ret_5d", "EMR_Emerson_ret_20d", "vix_max_abs_ret_5d__macross__heston_theta"], "is_new": false}, {"model_id": "v2_h5_NORMAL_GradientBoosting_OptunaCal_N19", "algo": "GradientBoostingCal", "regime": "NORMAL", "horizon": 5, "n_features": 19, "F1_dir": 0.5334, "F1_UP_FORT": 0.3333, "F1_DOWN_FORT": 0.4136, "train_start": "2001-02-06", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VIX_Price_zscore_60d__prod__CCI_CrownCastle_vol_20d", "VIX_Price_zscore_60d__div__heston_theta", "VIX_Price_zscore_60d__div__hmm_p_stress", "EWY_Korea_ret_20d", "M_Macys_vol_20d", "Core_CPI_zscore_60d__macross__VRP", "VIX_Price_zscore_60d__div__vix_max_abs_ret_5d", "NFCI_ret_5d", "vix_zscore_10d__div__vix_vol_of_vol_10d", "Nikkei_Japan_vol_20d", "AMT_AmericanTower_zscore_60d__minus__vix_level", "3M_vol_20d", "VIX_Price_zscore_60d__div__VRP", "VIX_Price_zscore_60d__div__MCD_ret_5d", "SO_SouthernCo_ret_5d", "vix_vol_of_vol_10d__prod__heston_theta", "heston_var_ev_h1__minus__MCD_ret_5d", "EMR_Emerson_ret_20d", "vix_max_abs_ret_5d__macross__heston_theta"], "is_new": false}, {"model_id": "v2_h5_STRESS_XGBoost_N8", "algo": "XGBoost", "regime": "STRESS", "horizon": 5, "n_features": 8, "F1_dir": 0.5088, "F1_UP_FORT": 0.2078, "F1_DOWN_FORT": 0.4354, "train_start": "2000-11-15", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VIX_Price_zscore_60d__prod__Core_PCE_zscore_60d", "hmm_p_stress", "SBUX_ret_20d__minus__VIX_Price_ret_5d", "AAPL_ret_5d__minus__vix_vol_of_vol_10d", "VIX_Price_zscore_60d__zrel__EWZ_Brazil_zscore_60d", "PAYX_Paychex_zscore_60d__ret5x__STLFSI4_zscore_60d", "AAPL_ret_5d__zrel__vix_vol_of_vol_10d", "DOW_Price_zscore_60d"], "is_new": false}, {"model_id": "v2_h5_STRESS_XGBoost_N9", "algo": "XGBoost", "regime": "STRESS", "horizon": 5, "n_features": 9, "F1_dir": 0.5048, "F1_UP_FORT": 0.2533, "F1_DOWN_FORT": 0.4615, "train_start": "2000-11-15", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VIX_Price_zscore_60d__prod__Core_PCE_zscore_60d", "hmm_p_stress", "SBUX_ret_20d__minus__VIX_Price_ret_5d", "AAPL_ret_5d__minus__vix_vol_of_vol_10d", "VIX_Price_zscore_60d__zrel__EWZ_Brazil_zscore_60d", "PAYX_Paychex_zscore_60d__ret5x__STLFSI4_zscore_60d", "AAPL_ret_5d__zrel__vix_vol_of_vol_10d", "DOW_Price_zscore_60d", "ORCL_zscore_60d"], "is_new": false}, {"model_id": "v2_h5_STRESS_XGBoost_N10", "algo": "XGBoost", "regime": "STRESS", "horizon": 5, "n_features": 10, "F1_dir": 0.5086, "F1_UP_FORT": 0.2785, "F1_DOWN_FORT": 0.4639, "train_start": "2000-11-15", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VIX_Price_zscore_60d__prod__Core_PCE_zscore_60d", "hmm_p_stress", "SBUX_ret_20d__minus__VIX_Price_ret_5d", "AAPL_ret_5d__minus__vix_vol_of_vol_10d", "VIX_Price_zscore_60d__zrel__EWZ_Brazil_zscore_60d", "PAYX_Paychex_zscore_60d__ret5x__STLFSI4_zscore_60d", "AAPL_ret_5d__zrel__vix_vol_of_vol_10d", "DOW_Price_zscore_60d", "ORCL_zscore_60d", "EWZ_Brazil_zscore_60d__zrel__STLFSI4_zscore_60d"], "is_new": false}, {"model_id": "v2_h5_STRESS_XGBoost_N12", "algo": "XGBoost", "regime": "STRESS", "horizon": 5, "n_features": 12, "F1_dir": 0.5295, "F1_UP_FORT": 0.3137, "F1_DOWN_FORT": 0.5376, "train_start": "2000-11-15", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VIX_Price_zscore_60d__prod__Core_PCE_zscore_60d", "hmm_p_stress", "SBUX_ret_20d__minus__VIX_Price_ret_5d", "AAPL_ret_5d__minus__vix_vol_of_vol_10d", "VIX_Price_zscore_60d__zrel__EWZ_Brazil_zscore_60d", "PAYX_Paychex_zscore_60d__ret5x__STLFSI4_zscore_60d", "AAPL_ret_5d__zrel__vix_vol_of_vol_10d", "DOW_Price_zscore_60d", "ORCL_zscore_60d", "EWZ_Brazil_zscore_60d__zrel__STLFSI4_zscore_60d", "LUV_SouthwestAir_zscore_60d__ret5x__NFCI_ret_5d", "VIX_Price_zscore_60d__macross__AMD_zscore_60d"], "is_new": false}, {"model_id": "v2_h5_STRESS_XGBoost_N13", "algo": "XGBoost", "regime": "STRESS", "horizon": 5, "n_features": 13, "F1_dir": 0.5194, "F1_UP_FORT": 0.3179, "F1_DOWN_FORT": 0.5247, "train_start": "2000-11-15", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VIX_Price_zscore_60d__prod__Core_PCE_zscore_60d", "hmm_p_stress", "SBUX_ret_20d__minus__VIX_Price_ret_5d", "AAPL_ret_5d__minus__vix_vol_of_vol_10d", "VIX_Price_zscore_60d__zrel__EWZ_Brazil_zscore_60d", "PAYX_Paychex_zscore_60d__ret5x__STLFSI4_zscore_60d", "AAPL_ret_5d__zrel__vix_vol_of_vol_10d", "DOW_Price_zscore_60d", "ORCL_zscore_60d", "EWZ_Brazil_zscore_60d__zrel__STLFSI4_zscore_60d", "LUV_SouthwestAir_zscore_60d__ret5x__NFCI_ret_5d", "VIX_Price_zscore_60d__macross__AMD_zscore_60d", "EWM_Malaysia_zscore_60d"], "is_new": false}, {"model_id": "v2_h5_STRESS_XGBoost_N14", "algo": "XGBoost", "regime": "STRESS", "horizon": 5, "n_features": 14, "F1_dir": 0.5276, "F1_UP_FORT": 0.2788, "F1_DOWN_FORT": 0.5149, "train_start": "2000-11-15", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VIX_Price_zscore_60d__prod__Core_PCE_zscore_60d", "hmm_p_stress", "SBUX_ret_20d__minus__VIX_Price_ret_5d", "AAPL_ret_5d__minus__vix_vol_of_vol_10d", "VIX_Price_zscore_60d__zrel__EWZ_Brazil_zscore_60d", "PAYX_Paychex_zscore_60d__ret5x__STLFSI4_zscore_60d", "AAPL_ret_5d__zrel__vix_vol_of_vol_10d", "DOW_Price_zscore_60d", "ORCL_zscore_60d", "EWZ_Brazil_zscore_60d__zrel__STLFSI4_zscore_60d", "LUV_SouthwestAir_zscore_60d__ret5x__NFCI_ret_5d", "VIX_Price_zscore_60d__macross__AMD_zscore_60d", "EWM_Malaysia_zscore_60d", "T_ret_1d"], "is_new": false}, {"model_id": "v2_h5_STRESS_XGBoost_N15", "algo": "XGBoost", "regime": "STRESS", "horizon": 5, "n_features": 15, "F1_dir": 0.5448, "F1_UP_FORT": 0.2933, "F1_DOWN_FORT": 0.5113, "train_start": "2000-11-15", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VIX_Price_zscore_60d__prod__Core_PCE_zscore_60d", "hmm_p_stress", "SBUX_ret_20d__minus__VIX_Price_ret_5d", "AAPL_ret_5d__minus__vix_vol_of_vol_10d", "VIX_Price_zscore_60d__zrel__EWZ_Brazil_zscore_60d", "PAYX_Paychex_zscore_60d__ret5x__STLFSI4_zscore_60d", "AAPL_ret_5d__zrel__vix_vol_of_vol_10d", "DOW_Price_zscore_60d", "ORCL_zscore_60d", "EWZ_Brazil_zscore_60d__zrel__STLFSI4_zscore_60d", "LUV_SouthwestAir_zscore_60d__ret5x__NFCI_ret_5d", "VIX_Price_zscore_60d__macross__AMD_zscore_60d", "EWM_Malaysia_zscore_60d", "T_ret_1d", "LUV_SouthwestAir_zscore_60d__div__PAYX_Paychex_zscore_60d"], "is_new": false}, {"model_id": "v2_h5_STRESS_XGBoost_N16", "algo": "XGBoost", "regime": "STRESS", "horizon": 5, "n_features": 16, "F1_dir": 0.5397, "F1_UP_FORT": 0.2892, "F1_DOWN_FORT": 0.5323, "train_start": "2000-11-15", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VIX_Price_zscore_60d__prod__Core_PCE_zscore_60d", "hmm_p_stress", "SBUX_ret_20d__minus__VIX_Price_ret_5d", "AAPL_ret_5d__minus__vix_vol_of_vol_10d", "VIX_Price_zscore_60d__zrel__EWZ_Brazil_zscore_60d", "PAYX_Paychex_zscore_60d__ret5x__STLFSI4_zscore_60d", "AAPL_ret_5d__zrel__vix_vol_of_vol_10d", "DOW_Price_zscore_60d", "ORCL_zscore_60d", "EWZ_Brazil_zscore_60d__zrel__STLFSI4_zscore_60d", "LUV_SouthwestAir_zscore_60d__ret5x__NFCI_ret_5d", "VIX_Price_zscore_60d__macross__AMD_zscore_60d", "EWM_Malaysia_zscore_60d", "T_ret_1d", "LUV_SouthwestAir_zscore_60d__div__PAYX_Paychex_zscore_60d", "EWT_Taiwan_ret_1d__ret5x__NFCI_ret_5d"], "is_new": false}, {"model_id": "v2_h5_STRESS_XGBoost_N17", "algo": "XGBoost", "regime": "STRESS", "horizon": 5, "n_features": 17, "F1_dir": 0.5426, "F1_UP_FORT": 0.2517, "F1_DOWN_FORT": 0.4981, "train_start": "2000-11-15", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VIX_Price_zscore_60d__prod__Core_PCE_zscore_60d", "hmm_p_stress", "SBUX_ret_20d__minus__VIX_Price_ret_5d", "AAPL_ret_5d__minus__vix_vol_of_vol_10d", "VIX_Price_zscore_60d__zrel__EWZ_Brazil_zscore_60d", "PAYX_Paychex_zscore_60d__ret5x__STLFSI4_zscore_60d", "AAPL_ret_5d__zrel__vix_vol_of_vol_10d", "DOW_Price_zscore_60d", "ORCL_zscore_60d", "EWZ_Brazil_zscore_60d__zrel__STLFSI4_zscore_60d", "LUV_SouthwestAir_zscore_60d__ret5x__NFCI_ret_5d", "VIX_Price_zscore_60d__macross__AMD_zscore_60d", "EWM_Malaysia_zscore_60d", "T_ret_1d", "LUV_SouthwestAir_zscore_60d__div__PAYX_Paychex_zscore_60d", "EWT_Taiwan_ret_1d__ret5x__NFCI_ret_5d", "GILD_Gilead_ret_20d"], "is_new": false}, {"model_id": "v2_h5_STRESS_XGBoost_N18", "algo": "XGBoost", "regime": "STRESS", "horizon": 5, "n_features": 18, "F1_dir": 0.5296, "F1_UP_FORT": 0.2535, "F1_DOWN_FORT": 0.5037, "train_start": "2000-11-15", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VIX_Price_zscore_60d__prod__Core_PCE_zscore_60d", "hmm_p_stress", "SBUX_ret_20d__minus__VIX_Price_ret_5d", "AAPL_ret_5d__minus__vix_vol_of_vol_10d", "VIX_Price_zscore_60d__zrel__EWZ_Brazil_zscore_60d", "PAYX_Paychex_zscore_60d__ret5x__STLFSI4_zscore_60d", "AAPL_ret_5d__zrel__vix_vol_of_vol_10d", "DOW_Price_zscore_60d", "ORCL_zscore_60d", "EWZ_Brazil_zscore_60d__zrel__STLFSI4_zscore_60d", "LUV_SouthwestAir_zscore_60d__ret5x__NFCI_ret_5d", "VIX_Price_zscore_60d__macross__AMD_zscore_60d", "EWM_Malaysia_zscore_60d", "T_ret_1d", "LUV_SouthwestAir_zscore_60d__div__PAYX_Paychex_zscore_60d", "EWT_Taiwan_ret_1d__ret5x__NFCI_ret_5d", "GILD_Gilead_ret_20d", "TGT_Target_zscore_60d"], "is_new": false}, {"model_id": "v2_h5_STRESS_XGBoost_N19", "algo": "XGBoost", "regime": "STRESS", "horizon": 5, "n_features": 19, "F1_dir": 0.5469, "F1_UP_FORT": 0.2482, "F1_DOWN_FORT": 0.5092, "train_start": "2000-11-15", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VIX_Price_zscore_60d__prod__Core_PCE_zscore_60d", "hmm_p_stress", "SBUX_ret_20d__minus__VIX_Price_ret_5d", "AAPL_ret_5d__minus__vix_vol_of_vol_10d", "VIX_Price_zscore_60d__zrel__EWZ_Brazil_zscore_60d", "PAYX_Paychex_zscore_60d__ret5x__STLFSI4_zscore_60d", "AAPL_ret_5d__zrel__vix_vol_of_vol_10d", "DOW_Price_zscore_60d", "ORCL_zscore_60d", "EWZ_Brazil_zscore_60d__zrel__STLFSI4_zscore_60d", "LUV_SouthwestAir_zscore_60d__ret5x__NFCI_ret_5d", "VIX_Price_zscore_60d__macross__AMD_zscore_60d", "EWM_Malaysia_zscore_60d", "T_ret_1d", "LUV_SouthwestAir_zscore_60d__div__PAYX_Paychex_zscore_60d", "EWT_Taiwan_ret_1d__ret5x__NFCI_ret_5d", "GILD_Gilead_ret_20d", "TGT_Target_zscore_60d", "NEE_NextEra_ret_20d"], "is_new": false}, {"model_id": "v2_h5_STRESS_XGBoost_N20", "algo": "XGBoost", "regime": "STRESS", "horizon": 5, "n_features": 20, "F1_dir": 0.5468, "F1_UP_FORT": 0.2703, "F1_DOWN_FORT": 0.518, "train_start": "2000-11-15", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VIX_Price_zscore_60d__prod__Core_PCE_zscore_60d", "hmm_p_stress", "SBUX_ret_20d__minus__VIX_Price_ret_5d", "AAPL_ret_5d__minus__vix_vol_of_vol_10d", "VIX_Price_zscore_60d__zrel__EWZ_Brazil_zscore_60d", "PAYX_Paychex_zscore_60d__ret5x__STLFSI4_zscore_60d", "AAPL_ret_5d__zrel__vix_vol_of_vol_10d", "DOW_Price_zscore_60d", "ORCL_zscore_60d", "EWZ_Brazil_zscore_60d__zrel__STLFSI4_zscore_60d", "LUV_SouthwestAir_zscore_60d__ret5x__NFCI_ret_5d", "VIX_Price_zscore_60d__macross__AMD_zscore_60d", "EWM_Malaysia_zscore_60d", "T_ret_1d", "LUV_SouthwestAir_zscore_60d__div__PAYX_Paychex_zscore_60d", "EWT_Taiwan_ret_1d__ret5x__NFCI_ret_5d", "GILD_Gilead_ret_20d", "TGT_Target_zscore_60d", "NEE_NextEra_ret_20d", "BDX_Becton_Dickinson_ret_20d"], "is_new": false}, {"model_id": "v2_h5_STRESS_LightGBM_N6", "algo": "LightGBM", "regime": "STRESS", "horizon": 5, "n_features": 6, "F1_dir": 0.5084, "F1_UP_FORT": 0.2252, "F1_DOWN_FORT": 0.4627, "train_start": "2000-11-15", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VIX_Price_zscore_60d__prod__Core_PCE_zscore_60d", "hmm_p_stress", "SBUX_ret_20d__minus__VIX_Price_ret_5d", "AAPL_ret_5d__minus__vix_vol_of_vol_10d", "VIX_Price_zscore_60d__zrel__EWZ_Brazil_zscore_60d", "PAYX_Paychex_zscore_60d__ret5x__STLFSI4_zscore_60d"], "is_new": false}, {"model_id": "v2_h5_STRESS_LightGBM_N10", "algo": "LightGBM", "regime": "STRESS", "horizon": 5, "n_features": 10, "F1_dir": 0.5153, "F1_UP_FORT": 0.2148, "F1_DOWN_FORT": 0.4496, "train_start": "2000-11-15", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VIX_Price_zscore_60d__prod__Core_PCE_zscore_60d", "hmm_p_stress", "SBUX_ret_20d__minus__VIX_Price_ret_5d", "AAPL_ret_5d__minus__vix_vol_of_vol_10d", "VIX_Price_zscore_60d__zrel__EWZ_Brazil_zscore_60d", "PAYX_Paychex_zscore_60d__ret5x__STLFSI4_zscore_60d", "AAPL_ret_5d__zrel__vix_vol_of_vol_10d", "DOW_Price_zscore_60d", "ORCL_zscore_60d", "EWZ_Brazil_zscore_60d__zrel__STLFSI4_zscore_60d"], "is_new": false}, {"model_id": "v2_h5_STRESS_LightGBM_N11", "algo": "LightGBM", "regime": "STRESS", "horizon": 5, "n_features": 11, "F1_dir": 0.5111, "F1_UP_FORT": 0.2133, "F1_DOWN_FORT": 0.5199, "train_start": "2000-11-15", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VIX_Price_zscore_60d__prod__Core_PCE_zscore_60d", "hmm_p_stress", "SBUX_ret_20d__minus__VIX_Price_ret_5d", "AAPL_ret_5d__minus__vix_vol_of_vol_10d", "VIX_Price_zscore_60d__zrel__EWZ_Brazil_zscore_60d", "PAYX_Paychex_zscore_60d__ret5x__STLFSI4_zscore_60d", "AAPL_ret_5d__zrel__vix_vol_of_vol_10d", "DOW_Price_zscore_60d", "ORCL_zscore_60d", "EWZ_Brazil_zscore_60d__zrel__STLFSI4_zscore_60d", "LUV_SouthwestAir_zscore_60d__ret5x__NFCI_ret_5d"], "is_new": false}, {"model_id": "v2_h5_STRESS_LightGBM_N12", "algo": "LightGBM", "regime": "STRESS", "horizon": 5, "n_features": 12, "F1_dir": 0.5173, "F1_UP_FORT": 0.2152, "F1_DOWN_FORT": 0.4943, "train_start": "2000-11-15", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VIX_Price_zscore_60d__prod__Core_PCE_zscore_60d", "hmm_p_stress", "SBUX_ret_20d__minus__VIX_Price_ret_5d", "AAPL_ret_5d__minus__vix_vol_of_vol_10d", "VIX_Price_zscore_60d__zrel__EWZ_Brazil_zscore_60d", "PAYX_Paychex_zscore_60d__ret5x__STLFSI4_zscore_60d", "AAPL_ret_5d__zrel__vix_vol_of_vol_10d", "DOW_Price_zscore_60d", "ORCL_zscore_60d", "EWZ_Brazil_zscore_60d__zrel__STLFSI4_zscore_60d", "LUV_SouthwestAir_zscore_60d__ret5x__NFCI_ret_5d", "VIX_Price_zscore_60d__macross__AMD_zscore_60d"], "is_new": false}, {"model_id": "v2_h5_STRESS_LightGBM_N13", "algo": "LightGBM", "regime": "STRESS", "horizon": 5, "n_features": 13, "F1_dir": 0.5274, "F1_UP_FORT": 0.2716, "F1_DOWN_FORT": 0.4841, "train_start": "2000-11-15", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VIX_Price_zscore_60d__prod__Core_PCE_zscore_60d", "hmm_p_stress", "SBUX_ret_20d__minus__VIX_Price_ret_5d", "AAPL_ret_5d__minus__vix_vol_of_vol_10d", "VIX_Price_zscore_60d__zrel__EWZ_Brazil_zscore_60d", "PAYX_Paychex_zscore_60d__ret5x__STLFSI4_zscore_60d", "AAPL_ret_5d__zrel__vix_vol_of_vol_10d", "DOW_Price_zscore_60d", "ORCL_zscore_60d", "EWZ_Brazil_zscore_60d__zrel__STLFSI4_zscore_60d", "LUV_SouthwestAir_zscore_60d__ret5x__NFCI_ret_5d", "VIX_Price_zscore_60d__macross__AMD_zscore_60d", "EWM_Malaysia_zscore_60d"], "is_new": false}, {"model_id": "v2_h5_STRESS_LightGBM_N14", "algo": "LightGBM", "regime": "STRESS", "horizon": 5, "n_features": 14, "F1_dir": 0.5481, "F1_UP_FORT": 0.284, "F1_DOWN_FORT": 0.5227, "train_start": "2000-11-15", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VIX_Price_zscore_60d__prod__Core_PCE_zscore_60d", "hmm_p_stress", "SBUX_ret_20d__minus__VIX_Price_ret_5d", "AAPL_ret_5d__minus__vix_vol_of_vol_10d", "VIX_Price_zscore_60d__zrel__EWZ_Brazil_zscore_60d", "PAYX_Paychex_zscore_60d__ret5x__STLFSI4_zscore_60d", "AAPL_ret_5d__zrel__vix_vol_of_vol_10d", "DOW_Price_zscore_60d", "ORCL_zscore_60d", "EWZ_Brazil_zscore_60d__zrel__STLFSI4_zscore_60d", "LUV_SouthwestAir_zscore_60d__ret5x__NFCI_ret_5d", "VIX_Price_zscore_60d__macross__AMD_zscore_60d", "EWM_Malaysia_zscore_60d", "T_ret_1d"], "is_new": false}, {"model_id": "v2_h5_STRESS_LightGBM_N15", "algo": "LightGBM", "regime": "STRESS", "horizon": 5, "n_features": 15, "F1_dir": 0.5194, "F1_UP_FORT": 0.2767, "F1_DOWN_FORT": 0.4867, "train_start": "2000-11-15", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VIX_Price_zscore_60d__prod__Core_PCE_zscore_60d", "hmm_p_stress", "SBUX_ret_20d__minus__VIX_Price_ret_5d", "AAPL_ret_5d__minus__vix_vol_of_vol_10d", "VIX_Price_zscore_60d__zrel__EWZ_Brazil_zscore_60d", "PAYX_Paychex_zscore_60d__ret5x__STLFSI4_zscore_60d", "AAPL_ret_5d__zrel__vix_vol_of_vol_10d", "DOW_Price_zscore_60d", "ORCL_zscore_60d", "EWZ_Brazil_zscore_60d__zrel__STLFSI4_zscore_60d", "LUV_SouthwestAir_zscore_60d__ret5x__NFCI_ret_5d", "VIX_Price_zscore_60d__macross__AMD_zscore_60d", "EWM_Malaysia_zscore_60d", "T_ret_1d", "LUV_SouthwestAir_zscore_60d__div__PAYX_Paychex_zscore_60d"], "is_new": false}, {"model_id": "v2_h5_STRESS_LightGBM_N16", "algo": "LightGBM", "regime": "STRESS", "horizon": 5, "n_features": 16, "F1_dir": 0.531, "F1_UP_FORT": 0.2857, "F1_DOWN_FORT": 0.5098, "train_start": "2000-11-15", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VIX_Price_zscore_60d__prod__Core_PCE_zscore_60d", "hmm_p_stress", "SBUX_ret_20d__minus__VIX_Price_ret_5d", "AAPL_ret_5d__minus__vix_vol_of_vol_10d", "VIX_Price_zscore_60d__zrel__EWZ_Brazil_zscore_60d", "PAYX_Paychex_zscore_60d__ret5x__STLFSI4_zscore_60d", "AAPL_ret_5d__zrel__vix_vol_of_vol_10d", "DOW_Price_zscore_60d", "ORCL_zscore_60d", "EWZ_Brazil_zscore_60d__zrel__STLFSI4_zscore_60d", "LUV_SouthwestAir_zscore_60d__ret5x__NFCI_ret_5d", "VIX_Price_zscore_60d__macross__AMD_zscore_60d", "EWM_Malaysia_zscore_60d", "T_ret_1d", "LUV_SouthwestAir_zscore_60d__div__PAYX_Paychex_zscore_60d", "EWT_Taiwan_ret_1d__ret5x__NFCI_ret_5d"], "is_new": false}, {"model_id": "v2_h5_STRESS_LightGBM_N17", "algo": "LightGBM", "regime": "STRESS", "horizon": 5, "n_features": 17, "F1_dir": 0.5237, "F1_UP_FORT": 0.225, "F1_DOWN_FORT": 0.4743, "train_start": "2000-11-15", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VIX_Price_zscore_60d__prod__Core_PCE_zscore_60d", "hmm_p_stress", "SBUX_ret_20d__minus__VIX_Price_ret_5d", "AAPL_ret_5d__minus__vix_vol_of_vol_10d", "VIX_Price_zscore_60d__zrel__EWZ_Brazil_zscore_60d", "PAYX_Paychex_zscore_60d__ret5x__STLFSI4_zscore_60d", "AAPL_ret_5d__zrel__vix_vol_of_vol_10d", "DOW_Price_zscore_60d", "ORCL_zscore_60d", "EWZ_Brazil_zscore_60d__zrel__STLFSI4_zscore_60d", "LUV_SouthwestAir_zscore_60d__ret5x__NFCI_ret_5d", "VIX_Price_zscore_60d__macross__AMD_zscore_60d", "EWM_Malaysia_zscore_60d", "T_ret_1d", "LUV_SouthwestAir_zscore_60d__div__PAYX_Paychex_zscore_60d", "EWT_Taiwan_ret_1d__ret5x__NFCI_ret_5d", "GILD_Gilead_ret_20d"], "is_new": false}, {"model_id": "v2_h5_STRESS_LightGBM_N18", "algo": "LightGBM", "regime": "STRESS", "horizon": 5, "n_features": 18, "F1_dir": 0.5577, "F1_UP_FORT": 0.2597, "F1_DOWN_FORT": 0.4961, "train_start": "2000-11-15", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VIX_Price_zscore_60d__prod__Core_PCE_zscore_60d", "hmm_p_stress", "SBUX_ret_20d__minus__VIX_Price_ret_5d", "AAPL_ret_5d__minus__vix_vol_of_vol_10d", "VIX_Price_zscore_60d__zrel__EWZ_Brazil_zscore_60d", "PAYX_Paychex_zscore_60d__ret5x__STLFSI4_zscore_60d", "AAPL_ret_5d__zrel__vix_vol_of_vol_10d", "DOW_Price_zscore_60d", "ORCL_zscore_60d", "EWZ_Brazil_zscore_60d__zrel__STLFSI4_zscore_60d", "LUV_SouthwestAir_zscore_60d__ret5x__NFCI_ret_5d", "VIX_Price_zscore_60d__macross__AMD_zscore_60d", "EWM_Malaysia_zscore_60d", "T_ret_1d", "LUV_SouthwestAir_zscore_60d__div__PAYX_Paychex_zscore_60d", "EWT_Taiwan_ret_1d__ret5x__NFCI_ret_5d", "GILD_Gilead_ret_20d", "TGT_Target_zscore_60d"], "is_new": false}, {"model_id": "v2_h5_STRESS_LightGBM_N19", "algo": "LightGBM", "regime": "STRESS", "horizon": 5, "n_features": 19, "F1_dir": 0.5223, "F1_UP_FORT": 0.194, "F1_DOWN_FORT": 0.5, "train_start": "2000-11-15", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VIX_Price_zscore_60d__prod__Core_PCE_zscore_60d", "hmm_p_stress", "SBUX_ret_20d__minus__VIX_Price_ret_5d", "AAPL_ret_5d__minus__vix_vol_of_vol_10d", "VIX_Price_zscore_60d__zrel__EWZ_Brazil_zscore_60d", "PAYX_Paychex_zscore_60d__ret5x__STLFSI4_zscore_60d", "AAPL_ret_5d__zrel__vix_vol_of_vol_10d", "DOW_Price_zscore_60d", "ORCL_zscore_60d", "EWZ_Brazil_zscore_60d__zrel__STLFSI4_zscore_60d", "LUV_SouthwestAir_zscore_60d__ret5x__NFCI_ret_5d", "VIX_Price_zscore_60d__macross__AMD_zscore_60d", "EWM_Malaysia_zscore_60d", "T_ret_1d", "LUV_SouthwestAir_zscore_60d__div__PAYX_Paychex_zscore_60d", "EWT_Taiwan_ret_1d__ret5x__NFCI_ret_5d", "GILD_Gilead_ret_20d", "TGT_Target_zscore_60d", "NEE_NextEra_ret_20d"], "is_new": false}, {"model_id": "v2_h5_STRESS_LightGBM_N20", "algo": "LightGBM", "regime": "STRESS", "horizon": 5, "n_features": 20, "F1_dir": 0.5334, "F1_UP_FORT": 0.2759, "F1_DOWN_FORT": 0.5092, "train_start": "2000-11-15", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VIX_Price_zscore_60d__prod__Core_PCE_zscore_60d", "hmm_p_stress", "SBUX_ret_20d__minus__VIX_Price_ret_5d", "AAPL_ret_5d__minus__vix_vol_of_vol_10d", "VIX_Price_zscore_60d__zrel__EWZ_Brazil_zscore_60d", "PAYX_Paychex_zscore_60d__ret5x__STLFSI4_zscore_60d", "AAPL_ret_5d__zrel__vix_vol_of_vol_10d", "DOW_Price_zscore_60d", "ORCL_zscore_60d", "EWZ_Brazil_zscore_60d__zrel__STLFSI4_zscore_60d", "LUV_SouthwestAir_zscore_60d__ret5x__NFCI_ret_5d", "VIX_Price_zscore_60d__macross__AMD_zscore_60d", "EWM_Malaysia_zscore_60d", "T_ret_1d", "LUV_SouthwestAir_zscore_60d__div__PAYX_Paychex_zscore_60d", "EWT_Taiwan_ret_1d__ret5x__NFCI_ret_5d", "GILD_Gilead_ret_20d", "TGT_Target_zscore_60d", "NEE_NextEra_ret_20d", "BDX_Becton_Dickinson_ret_20d"], "is_new": false}, {"model_id": "v2_h5_STRESS_GradientBoosting_N11", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 5, "n_features": 11, "F1_dir": 0.5067, "F1_UP_FORT": 0.271, "F1_DOWN_FORT": 0.4697, "train_start": "2000-11-15", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VIX_Price_zscore_60d__prod__Core_PCE_zscore_60d", "hmm_p_stress", "SBUX_ret_20d__minus__VIX_Price_ret_5d", "AAPL_ret_5d__minus__vix_vol_of_vol_10d", "VIX_Price_zscore_60d__zrel__EWZ_Brazil_zscore_60d", "PAYX_Paychex_zscore_60d__ret5x__STLFSI4_zscore_60d", "AAPL_ret_5d__zrel__vix_vol_of_vol_10d", "DOW_Price_zscore_60d", "ORCL_zscore_60d", "EWZ_Brazil_zscore_60d__zrel__STLFSI4_zscore_60d", "LUV_SouthwestAir_zscore_60d__ret5x__NFCI_ret_5d"], "is_new": false}, {"model_id": "v2_h5_STRESS_GradientBoosting_N12", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 5, "n_features": 12, "F1_dir": 0.5028, "F1_UP_FORT": 0.2368, "F1_DOWN_FORT": 0.5128, "train_start": "2000-11-15", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VIX_Price_zscore_60d__prod__Core_PCE_zscore_60d", "hmm_p_stress", "SBUX_ret_20d__minus__VIX_Price_ret_5d", "AAPL_ret_5d__minus__vix_vol_of_vol_10d", "VIX_Price_zscore_60d__zrel__EWZ_Brazil_zscore_60d", "PAYX_Paychex_zscore_60d__ret5x__STLFSI4_zscore_60d", "AAPL_ret_5d__zrel__vix_vol_of_vol_10d", "DOW_Price_zscore_60d", "ORCL_zscore_60d", "EWZ_Brazil_zscore_60d__zrel__STLFSI4_zscore_60d", "LUV_SouthwestAir_zscore_60d__ret5x__NFCI_ret_5d", "VIX_Price_zscore_60d__macross__AMD_zscore_60d"], "is_new": false}, {"model_id": "v2_h5_STRESS_GradientBoosting_N13", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 5, "n_features": 13, "F1_dir": 0.5069, "F1_UP_FORT": 0.2981, "F1_DOWN_FORT": 0.5113, "train_start": "2000-11-15", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VIX_Price_zscore_60d__prod__Core_PCE_zscore_60d", "hmm_p_stress", "SBUX_ret_20d__minus__VIX_Price_ret_5d", "AAPL_ret_5d__minus__vix_vol_of_vol_10d", "VIX_Price_zscore_60d__zrel__EWZ_Brazil_zscore_60d", "PAYX_Paychex_zscore_60d__ret5x__STLFSI4_zscore_60d", "AAPL_ret_5d__zrel__vix_vol_of_vol_10d", "DOW_Price_zscore_60d", "ORCL_zscore_60d", "EWZ_Brazil_zscore_60d__zrel__STLFSI4_zscore_60d", "LUV_SouthwestAir_zscore_60d__ret5x__NFCI_ret_5d", "VIX_Price_zscore_60d__macross__AMD_zscore_60d", "EWM_Malaysia_zscore_60d"], "is_new": false}, {"model_id": "v2_h5_STRESS_GradientBoosting_N14", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 5, "n_features": 14, "F1_dir": 0.5132, "F1_UP_FORT": 0.2484, "F1_DOWN_FORT": 0.4906, "train_start": "2000-11-15", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VIX_Price_zscore_60d__prod__Core_PCE_zscore_60d", "hmm_p_stress", "SBUX_ret_20d__minus__VIX_Price_ret_5d", "AAPL_ret_5d__minus__vix_vol_of_vol_10d", "VIX_Price_zscore_60d__zrel__EWZ_Brazil_zscore_60d", "PAYX_Paychex_zscore_60d__ret5x__STLFSI4_zscore_60d", "AAPL_ret_5d__zrel__vix_vol_of_vol_10d", "DOW_Price_zscore_60d", "ORCL_zscore_60d", "EWZ_Brazil_zscore_60d__zrel__STLFSI4_zscore_60d", "LUV_SouthwestAir_zscore_60d__ret5x__NFCI_ret_5d", "VIX_Price_zscore_60d__macross__AMD_zscore_60d", "EWM_Malaysia_zscore_60d", "T_ret_1d"], "is_new": false}, {"model_id": "v2_h5_STRESS_GradientBoosting_N15", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 5, "n_features": 15, "F1_dir": 0.555, "F1_UP_FORT": 0.3152, "F1_DOWN_FORT": 0.4962, "train_start": "2000-11-15", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VIX_Price_zscore_60d__prod__Core_PCE_zscore_60d", "hmm_p_stress", "SBUX_ret_20d__minus__VIX_Price_ret_5d", "AAPL_ret_5d__minus__vix_vol_of_vol_10d", "VIX_Price_zscore_60d__zrel__EWZ_Brazil_zscore_60d", "PAYX_Paychex_zscore_60d__ret5x__STLFSI4_zscore_60d", "AAPL_ret_5d__zrel__vix_vol_of_vol_10d", "DOW_Price_zscore_60d", "ORCL_zscore_60d", "EWZ_Brazil_zscore_60d__zrel__STLFSI4_zscore_60d", "LUV_SouthwestAir_zscore_60d__ret5x__NFCI_ret_5d", "VIX_Price_zscore_60d__macross__AMD_zscore_60d", "EWM_Malaysia_zscore_60d", "T_ret_1d", "LUV_SouthwestAir_zscore_60d__div__PAYX_Paychex_zscore_60d"], "is_new": false}, {"model_id": "v2_h5_STRESS_GradientBoosting_N16", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 5, "n_features": 16, "F1_dir": 0.5521, "F1_UP_FORT": 0.3095, "F1_DOWN_FORT": 0.4981, "train_start": "2000-11-15", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VIX_Price_zscore_60d__prod__Core_PCE_zscore_60d", "hmm_p_stress", "SBUX_ret_20d__minus__VIX_Price_ret_5d", "AAPL_ret_5d__minus__vix_vol_of_vol_10d", "VIX_Price_zscore_60d__zrel__EWZ_Brazil_zscore_60d", "PAYX_Paychex_zscore_60d__ret5x__STLFSI4_zscore_60d", "AAPL_ret_5d__zrel__vix_vol_of_vol_10d", "DOW_Price_zscore_60d", "ORCL_zscore_60d", "EWZ_Brazil_zscore_60d__zrel__STLFSI4_zscore_60d", "LUV_SouthwestAir_zscore_60d__ret5x__NFCI_ret_5d", "VIX_Price_zscore_60d__macross__AMD_zscore_60d", "EWM_Malaysia_zscore_60d", "T_ret_1d", "LUV_SouthwestAir_zscore_60d__div__PAYX_Paychex_zscore_60d", "EWT_Taiwan_ret_1d__ret5x__NFCI_ret_5d"], "is_new": false}, {"model_id": "v2_h5_STRESS_GradientBoosting_N17", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 5, "n_features": 17, "F1_dir": 0.5565, "F1_UP_FORT": 0.2927, "F1_DOWN_FORT": 0.4846, "train_start": "2000-11-15", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VIX_Price_zscore_60d__prod__Core_PCE_zscore_60d", "hmm_p_stress", "SBUX_ret_20d__minus__VIX_Price_ret_5d", "AAPL_ret_5d__minus__vix_vol_of_vol_10d", "VIX_Price_zscore_60d__zrel__EWZ_Brazil_zscore_60d", "PAYX_Paychex_zscore_60d__ret5x__STLFSI4_zscore_60d", "AAPL_ret_5d__zrel__vix_vol_of_vol_10d", "DOW_Price_zscore_60d", "ORCL_zscore_60d", "EWZ_Brazil_zscore_60d__zrel__STLFSI4_zscore_60d", "LUV_SouthwestAir_zscore_60d__ret5x__NFCI_ret_5d", "VIX_Price_zscore_60d__macross__AMD_zscore_60d", "EWM_Malaysia_zscore_60d", "T_ret_1d", "LUV_SouthwestAir_zscore_60d__div__PAYX_Paychex_zscore_60d", "EWT_Taiwan_ret_1d__ret5x__NFCI_ret_5d", "GILD_Gilead_ret_20d"], "is_new": false}, {"model_id": "v2_h5_STRESS_GradientBoosting_N18", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 5, "n_features": 18, "F1_dir": 0.5448, "F1_UP_FORT": 0.2517, "F1_DOWN_FORT": 0.4904, "train_start": "2000-11-15", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VIX_Price_zscore_60d__prod__Core_PCE_zscore_60d", "hmm_p_stress", "SBUX_ret_20d__minus__VIX_Price_ret_5d", "AAPL_ret_5d__minus__vix_vol_of_vol_10d", "VIX_Price_zscore_60d__zrel__EWZ_Brazil_zscore_60d", "PAYX_Paychex_zscore_60d__ret5x__STLFSI4_zscore_60d", "AAPL_ret_5d__zrel__vix_vol_of_vol_10d", "DOW_Price_zscore_60d", "ORCL_zscore_60d", "EWZ_Brazil_zscore_60d__zrel__STLFSI4_zscore_60d", "LUV_SouthwestAir_zscore_60d__ret5x__NFCI_ret_5d", "VIX_Price_zscore_60d__macross__AMD_zscore_60d", "EWM_Malaysia_zscore_60d", "T_ret_1d", "LUV_SouthwestAir_zscore_60d__div__PAYX_Paychex_zscore_60d", "EWT_Taiwan_ret_1d__ret5x__NFCI_ret_5d", "GILD_Gilead_ret_20d", "TGT_Target_zscore_60d"], "is_new": false}, {"model_id": "v2_h5_STRESS_GradientBoosting_N19", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 5, "n_features": 19, "F1_dir": 0.5254, "F1_UP_FORT": 0.2222, "F1_DOWN_FORT": 0.4853, "train_start": "2000-11-15", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VIX_Price_zscore_60d__prod__Core_PCE_zscore_60d", "hmm_p_stress", "SBUX_ret_20d__minus__VIX_Price_ret_5d", "AAPL_ret_5d__minus__vix_vol_of_vol_10d", "VIX_Price_zscore_60d__zrel__EWZ_Brazil_zscore_60d", "PAYX_Paychex_zscore_60d__ret5x__STLFSI4_zscore_60d", "AAPL_ret_5d__zrel__vix_vol_of_vol_10d", "DOW_Price_zscore_60d", "ORCL_zscore_60d", "EWZ_Brazil_zscore_60d__zrel__STLFSI4_zscore_60d", "LUV_SouthwestAir_zscore_60d__ret5x__NFCI_ret_5d", "VIX_Price_zscore_60d__macross__AMD_zscore_60d", "EWM_Malaysia_zscore_60d", "T_ret_1d", "LUV_SouthwestAir_zscore_60d__div__PAYX_Paychex_zscore_60d", "EWT_Taiwan_ret_1d__ret5x__NFCI_ret_5d", "GILD_Gilead_ret_20d", "TGT_Target_zscore_60d", "NEE_NextEra_ret_20d"], "is_new": false}, {"model_id": "v2_h5_STRESS_GradientBoosting_N20", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 5, "n_features": 20, "F1_dir": 0.5508, "F1_UP_FORT": 0.2877, "F1_DOWN_FORT": 0.4779, "train_start": "2000-11-15", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VIX_Price_zscore_60d__prod__Core_PCE_zscore_60d", "hmm_p_stress", "SBUX_ret_20d__minus__VIX_Price_ret_5d", "AAPL_ret_5d__minus__vix_vol_of_vol_10d", "VIX_Price_zscore_60d__zrel__EWZ_Brazil_zscore_60d", "PAYX_Paychex_zscore_60d__ret5x__STLFSI4_zscore_60d", "AAPL_ret_5d__zrel__vix_vol_of_vol_10d", "DOW_Price_zscore_60d", "ORCL_zscore_60d", "EWZ_Brazil_zscore_60d__zrel__STLFSI4_zscore_60d", "LUV_SouthwestAir_zscore_60d__ret5x__NFCI_ret_5d", "VIX_Price_zscore_60d__macross__AMD_zscore_60d", "EWM_Malaysia_zscore_60d", "T_ret_1d", "LUV_SouthwestAir_zscore_60d__div__PAYX_Paychex_zscore_60d", "EWT_Taiwan_ret_1d__ret5x__NFCI_ret_5d", "GILD_Gilead_ret_20d", "TGT_Target_zscore_60d", "NEE_NextEra_ret_20d", "BDX_Becton_Dickinson_ret_20d"], "is_new": false}, {"model_id": "v2_h5_STRESS_RandomForest_N8", "algo": "RandomForest", "regime": "STRESS", "horizon": 5, "n_features": 8, "F1_dir": 0.509, "F1_UP_FORT": 0.2321, "F1_DOWN_FORT": 0.5346, "train_start": "2000-11-15", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VIX_Price_zscore_60d__prod__Core_PCE_zscore_60d", "hmm_p_stress", "SBUX_ret_20d__minus__VIX_Price_ret_5d", "AAPL_ret_5d__minus__vix_vol_of_vol_10d", "VIX_Price_zscore_60d__zrel__EWZ_Brazil_zscore_60d", "PAYX_Paychex_zscore_60d__ret5x__STLFSI4_zscore_60d", "AAPL_ret_5d__zrel__vix_vol_of_vol_10d", "DOW_Price_zscore_60d"], "is_new": false}, {"model_id": "v2_h5_STRESS_RandomForest_N9", "algo": "RandomForest", "regime": "STRESS", "horizon": 5, "n_features": 9, "F1_dir": 0.511, "F1_UP_FORT": 0.2881, "F1_DOWN_FORT": 0.5385, "train_start": "2000-11-15", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VIX_Price_zscore_60d__prod__Core_PCE_zscore_60d", "hmm_p_stress", "SBUX_ret_20d__minus__VIX_Price_ret_5d", "AAPL_ret_5d__minus__vix_vol_of_vol_10d", "VIX_Price_zscore_60d__zrel__EWZ_Brazil_zscore_60d", "PAYX_Paychex_zscore_60d__ret5x__STLFSI4_zscore_60d", "AAPL_ret_5d__zrel__vix_vol_of_vol_10d", "DOW_Price_zscore_60d", "ORCL_zscore_60d"], "is_new": false}, {"model_id": "v2_h5_STRESS_RandomForest_N10", "algo": "RandomForest", "regime": "STRESS", "horizon": 5, "n_features": 10, "F1_dir": 0.5098, "F1_UP_FORT": 0.2344, "F1_DOWN_FORT": 0.5548, "train_start": "2000-11-15", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VIX_Price_zscore_60d__prod__Core_PCE_zscore_60d", "hmm_p_stress", "SBUX_ret_20d__minus__VIX_Price_ret_5d", "AAPL_ret_5d__minus__vix_vol_of_vol_10d", "VIX_Price_zscore_60d__zrel__EWZ_Brazil_zscore_60d", "PAYX_Paychex_zscore_60d__ret5x__STLFSI4_zscore_60d", "AAPL_ret_5d__zrel__vix_vol_of_vol_10d", "DOW_Price_zscore_60d", "ORCL_zscore_60d", "EWZ_Brazil_zscore_60d__zrel__STLFSI4_zscore_60d"], "is_new": false}, {"model_id": "v2_h5_STRESS_RandomForest_N12", "algo": "RandomForest", "regime": "STRESS", "horizon": 5, "n_features": 12, "F1_dir": 0.5003, "F1_UP_FORT": 0.2479, "F1_DOWN_FORT": 0.5402, "train_start": "2000-11-15", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VIX_Price_zscore_60d__prod__Core_PCE_zscore_60d", "hmm_p_stress", "SBUX_ret_20d__minus__VIX_Price_ret_5d", "AAPL_ret_5d__minus__vix_vol_of_vol_10d", "VIX_Price_zscore_60d__zrel__EWZ_Brazil_zscore_60d", "PAYX_Paychex_zscore_60d__ret5x__STLFSI4_zscore_60d", "AAPL_ret_5d__zrel__vix_vol_of_vol_10d", "DOW_Price_zscore_60d", "ORCL_zscore_60d", "EWZ_Brazil_zscore_60d__zrel__STLFSI4_zscore_60d", "LUV_SouthwestAir_zscore_60d__ret5x__NFCI_ret_5d", "VIX_Price_zscore_60d__macross__AMD_zscore_60d"], "is_new": false}, {"model_id": "v2_h5_STRESS_RandomForest_N16", "algo": "RandomForest", "regime": "STRESS", "horizon": 5, "n_features": 16, "F1_dir": 0.5237, "F1_UP_FORT": 0.1667, "F1_DOWN_FORT": 0.5351, "train_start": "2000-11-15", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VIX_Price_zscore_60d__prod__Core_PCE_zscore_60d", "hmm_p_stress", "SBUX_ret_20d__minus__VIX_Price_ret_5d", "AAPL_ret_5d__minus__vix_vol_of_vol_10d", "VIX_Price_zscore_60d__zrel__EWZ_Brazil_zscore_60d", "PAYX_Paychex_zscore_60d__ret5x__STLFSI4_zscore_60d", "AAPL_ret_5d__zrel__vix_vol_of_vol_10d", "DOW_Price_zscore_60d", "ORCL_zscore_60d", "EWZ_Brazil_zscore_60d__zrel__STLFSI4_zscore_60d", "LUV_SouthwestAir_zscore_60d__ret5x__NFCI_ret_5d", "VIX_Price_zscore_60d__macross__AMD_zscore_60d", "EWM_Malaysia_zscore_60d", "T_ret_1d", "LUV_SouthwestAir_zscore_60d__div__PAYX_Paychex_zscore_60d", "EWT_Taiwan_ret_1d__ret5x__NFCI_ret_5d"], "is_new": false}, {"model_id": "v2_h5_STRESS_RandomForest_N17", "algo": "RandomForest", "regime": "STRESS", "horizon": 5, "n_features": 17, "F1_dir": 0.5138, "F1_UP_FORT": 0.1594, "F1_DOWN_FORT": 0.5188, "train_start": "2000-11-15", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VIX_Price_zscore_60d__prod__Core_PCE_zscore_60d", "hmm_p_stress", "SBUX_ret_20d__minus__VIX_Price_ret_5d", "AAPL_ret_5d__minus__vix_vol_of_vol_10d", "VIX_Price_zscore_60d__zrel__EWZ_Brazil_zscore_60d", "PAYX_Paychex_zscore_60d__ret5x__STLFSI4_zscore_60d", "AAPL_ret_5d__zrel__vix_vol_of_vol_10d", "DOW_Price_zscore_60d", "ORCL_zscore_60d", "EWZ_Brazil_zscore_60d__zrel__STLFSI4_zscore_60d", "LUV_SouthwestAir_zscore_60d__ret5x__NFCI_ret_5d", "VIX_Price_zscore_60d__macross__AMD_zscore_60d", "EWM_Malaysia_zscore_60d", "T_ret_1d", "LUV_SouthwestAir_zscore_60d__div__PAYX_Paychex_zscore_60d", "EWT_Taiwan_ret_1d__ret5x__NFCI_ret_5d", "GILD_Gilead_ret_20d"], "is_new": false}, {"model_id": "v2_h5_STRESS_RandomForest_N18", "algo": "RandomForest", "regime": "STRESS", "horizon": 5, "n_features": 18, "F1_dir": 0.5207, "F1_UP_FORT": 0.1593, "F1_DOWN_FORT": 0.5246, "train_start": "2000-11-15", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VIX_Price_zscore_60d__prod__Core_PCE_zscore_60d", "hmm_p_stress", "SBUX_ret_20d__minus__VIX_Price_ret_5d", "AAPL_ret_5d__minus__vix_vol_of_vol_10d", "VIX_Price_zscore_60d__zrel__EWZ_Brazil_zscore_60d", "PAYX_Paychex_zscore_60d__ret5x__STLFSI4_zscore_60d", "AAPL_ret_5d__zrel__vix_vol_of_vol_10d", "DOW_Price_zscore_60d", "ORCL_zscore_60d", "EWZ_Brazil_zscore_60d__zrel__STLFSI4_zscore_60d", "LUV_SouthwestAir_zscore_60d__ret5x__NFCI_ret_5d", "VIX_Price_zscore_60d__macross__AMD_zscore_60d", "EWM_Malaysia_zscore_60d", "T_ret_1d", "LUV_SouthwestAir_zscore_60d__div__PAYX_Paychex_zscore_60d", "EWT_Taiwan_ret_1d__ret5x__NFCI_ret_5d", "GILD_Gilead_ret_20d", "TGT_Target_zscore_60d"], "is_new": false}, {"model_id": "v2_h5_STRESS_RandomForest_N19", "algo": "RandomForest", "regime": "STRESS", "horizon": 5, "n_features": 19, "F1_dir": 0.5139, "F1_UP_FORT": 0.1587, "F1_DOWN_FORT": 0.5246, "train_start": "2000-11-15", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VIX_Price_zscore_60d__prod__Core_PCE_zscore_60d", "hmm_p_stress", "SBUX_ret_20d__minus__VIX_Price_ret_5d", "AAPL_ret_5d__minus__vix_vol_of_vol_10d", "VIX_Price_zscore_60d__zrel__EWZ_Brazil_zscore_60d", "PAYX_Paychex_zscore_60d__ret5x__STLFSI4_zscore_60d", "AAPL_ret_5d__zrel__vix_vol_of_vol_10d", "DOW_Price_zscore_60d", "ORCL_zscore_60d", "EWZ_Brazil_zscore_60d__zrel__STLFSI4_zscore_60d", "LUV_SouthwestAir_zscore_60d__ret5x__NFCI_ret_5d", "VIX_Price_zscore_60d__macross__AMD_zscore_60d", "EWM_Malaysia_zscore_60d", "T_ret_1d", "LUV_SouthwestAir_zscore_60d__div__PAYX_Paychex_zscore_60d", "EWT_Taiwan_ret_1d__ret5x__NFCI_ret_5d", "GILD_Gilead_ret_20d", "TGT_Target_zscore_60d", "NEE_NextEra_ret_20d"], "is_new": false}, {"model_id": "v2_h5_STRESS_RandomForest_N20", "algo": "RandomForest", "regime": "STRESS", "horizon": 5, "n_features": 20, "F1_dir": 0.504, "F1_UP_FORT": 0.1221, "F1_DOWN_FORT": 0.5115, "train_start": "2000-11-15", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VIX_Price_zscore_60d__prod__Core_PCE_zscore_60d", "hmm_p_stress", "SBUX_ret_20d__minus__VIX_Price_ret_5d", "AAPL_ret_5d__minus__vix_vol_of_vol_10d", "VIX_Price_zscore_60d__zrel__EWZ_Brazil_zscore_60d", "PAYX_Paychex_zscore_60d__ret5x__STLFSI4_zscore_60d", "AAPL_ret_5d__zrel__vix_vol_of_vol_10d", "DOW_Price_zscore_60d", "ORCL_zscore_60d", "EWZ_Brazil_zscore_60d__zrel__STLFSI4_zscore_60d", "LUV_SouthwestAir_zscore_60d__ret5x__NFCI_ret_5d", "VIX_Price_zscore_60d__macross__AMD_zscore_60d", "EWM_Malaysia_zscore_60d", "T_ret_1d", "LUV_SouthwestAir_zscore_60d__div__PAYX_Paychex_zscore_60d", "EWT_Taiwan_ret_1d__ret5x__NFCI_ret_5d", "GILD_Gilead_ret_20d", "TGT_Target_zscore_60d", "NEE_NextEra_ret_20d", "BDX_Becton_Dickinson_ret_20d"], "is_new": false}, {"model_id": "v2_h5_STRESS_LogisticRegression_N5", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 5, "n_features": 5, "F1_dir": 0.5086, "F1_UP_FORT": 0.2945, "F1_DOWN_FORT": 0.5493, "train_start": "2000-11-15", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VIX_Price_zscore_60d__prod__Core_PCE_zscore_60d", "hmm_p_stress", "SBUX_ret_20d__minus__VIX_Price_ret_5d", "AAPL_ret_5d__minus__vix_vol_of_vol_10d", "VIX_Price_zscore_60d__zrel__EWZ_Brazil_zscore_60d"], "is_new": false}, {"model_id": "v2_h5_STRESS_LogisticRegression_N6", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 5, "n_features": 6, "F1_dir": 0.5166, "F1_UP_FORT": 0.3067, "F1_DOWN_FORT": 0.5614, "train_start": "2000-11-15", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VIX_Price_zscore_60d__prod__Core_PCE_zscore_60d", "hmm_p_stress", "SBUX_ret_20d__minus__VIX_Price_ret_5d", "AAPL_ret_5d__minus__vix_vol_of_vol_10d", "VIX_Price_zscore_60d__zrel__EWZ_Brazil_zscore_60d", "PAYX_Paychex_zscore_60d__ret5x__STLFSI4_zscore_60d"], "is_new": false}, {"model_id": "v2_h5_STRESS_LogisticRegression_N13", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 5, "n_features": 13, "F1_dir": 0.5046, "F1_UP_FORT": 0.183, "F1_DOWN_FORT": 0.5196, "train_start": "2000-11-15", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VIX_Price_zscore_60d__prod__Core_PCE_zscore_60d", "hmm_p_stress", "SBUX_ret_20d__minus__VIX_Price_ret_5d", "AAPL_ret_5d__minus__vix_vol_of_vol_10d", "VIX_Price_zscore_60d__zrel__EWZ_Brazil_zscore_60d", "PAYX_Paychex_zscore_60d__ret5x__STLFSI4_zscore_60d", "AAPL_ret_5d__zrel__vix_vol_of_vol_10d", "DOW_Price_zscore_60d", "ORCL_zscore_60d", "EWZ_Brazil_zscore_60d__zrel__STLFSI4_zscore_60d", "LUV_SouthwestAir_zscore_60d__ret5x__NFCI_ret_5d", "VIX_Price_zscore_60d__macross__AMD_zscore_60d", "EWM_Malaysia_zscore_60d"], "is_new": false}, {"model_id": "v2_h5_STRESS_XGBoost_Optuna_N15", "algo": "XGBoost", "regime": "STRESS", "horizon": 5, "n_features": 15, "F1_dir": 0.5389, "F1_UP_FORT": 0.3057, "F1_DOWN_FORT": 0.5159, "train_start": "2000-11-15", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VIX_Price_zscore_60d__prod__Core_PCE_zscore_60d", "hmm_p_stress", "SBUX_ret_20d__minus__VIX_Price_ret_5d", "AAPL_ret_5d__minus__vix_vol_of_vol_10d", "VIX_Price_zscore_60d__zrel__EWZ_Brazil_zscore_60d", "PAYX_Paychex_zscore_60d__ret5x__STLFSI4_zscore_60d", "AAPL_ret_5d__zrel__vix_vol_of_vol_10d", "DOW_Price_zscore_60d", "ORCL_zscore_60d", "EWZ_Brazil_zscore_60d__zrel__STLFSI4_zscore_60d", "LUV_SouthwestAir_zscore_60d__ret5x__NFCI_ret_5d", "VIX_Price_zscore_60d__macross__AMD_zscore_60d", "EWM_Malaysia_zscore_60d", "T_ret_1d", "LUV_SouthwestAir_zscore_60d__div__PAYX_Paychex_zscore_60d"], "is_new": false}, {"model_id": "v2_h5_STRESS_XGBoost_OptunaCal_N15", "algo": "XGBoostCal", "regime": "STRESS", "horizon": 5, "n_features": 15, "F1_dir": 0.5269, "F1_UP_FORT": 0.14, "F1_DOWN_FORT": 0.456, "train_start": "2000-11-15", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VIX_Price_zscore_60d__prod__Core_PCE_zscore_60d", "hmm_p_stress", "SBUX_ret_20d__minus__VIX_Price_ret_5d", "AAPL_ret_5d__minus__vix_vol_of_vol_10d", "VIX_Price_zscore_60d__zrel__EWZ_Brazil_zscore_60d", "PAYX_Paychex_zscore_60d__ret5x__STLFSI4_zscore_60d", "AAPL_ret_5d__zrel__vix_vol_of_vol_10d", "DOW_Price_zscore_60d", "ORCL_zscore_60d", "EWZ_Brazil_zscore_60d__zrel__STLFSI4_zscore_60d", "LUV_SouthwestAir_zscore_60d__ret5x__NFCI_ret_5d", "VIX_Price_zscore_60d__macross__AMD_zscore_60d", "EWM_Malaysia_zscore_60d", "T_ret_1d", "LUV_SouthwestAir_zscore_60d__div__PAYX_Paychex_zscore_60d"], "is_new": false}, {"model_id": "v2_h5_GLOBAL_XGBoost_N5", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 5, "n_features": 5, "F1_dir": 0.5213, "F1_UP_FORT": 0.2147, "F1_DOWN_FORT": 0.4144, "train_start": "2001-02-06", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["HON_Honeywell_zscore_60d__zrel__vix_vs_ma20", "VIX_Price_zscore_60d__div__NFCI_vol_20d", "vix_vs_ma10__zrel__US30Y_Rate_ret_20d", "PAYX_Paychex_vol_20d", "heston_var_ev_h7"], "is_new": false}, {"model_id": "v2_h5_GLOBAL_XGBoost_N6", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 5, "n_features": 6, "F1_dir": 0.5016, "F1_UP_FORT": 0.2, "F1_DOWN_FORT": 0.4241, "train_start": "2001-02-06", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["HON_Honeywell_zscore_60d__zrel__vix_vs_ma20", "VIX_Price_zscore_60d__div__NFCI_vol_20d", "vix_vs_ma10__zrel__US30Y_Rate_ret_20d", "PAYX_Paychex_vol_20d", "heston_var_ev_h7", "EWL_Switzerland_zscore_60d"], "is_new": false}, {"model_id": "v2_h5_GLOBAL_XGBoost_N7", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 5, "n_features": 7, "F1_dir": 0.5071, "F1_UP_FORT": 0.2191, "F1_DOWN_FORT": 0.4098, "train_start": "2001-02-06", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["HON_Honeywell_zscore_60d__zrel__vix_vs_ma20", "VIX_Price_zscore_60d__div__NFCI_vol_20d", "vix_vs_ma10__zrel__US30Y_Rate_ret_20d", "PAYX_Paychex_vol_20d", "heston_var_ev_h7", "EWL_Switzerland_zscore_60d", "HON_Honeywell_zscore_60d__minus__HD_zscore_60d"], "is_new": false}, {"model_id": "v2_h5_GLOBAL_XGBoost_N8", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 5, "n_features": 8, "F1_dir": 0.5029, "F1_UP_FORT": 0.2055, "F1_DOWN_FORT": 0.4248, "train_start": "2001-02-06", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["HON_Honeywell_zscore_60d__zrel__vix_vs_ma20", "VIX_Price_zscore_60d__div__NFCI_vol_20d", "vix_vs_ma10__zrel__US30Y_Rate_ret_20d", "PAYX_Paychex_vol_20d", "heston_var_ev_h7", "EWL_Switzerland_zscore_60d", "HON_Honeywell_zscore_60d__minus__HD_zscore_60d", "vix_zscore_10d__div__vix_vol_of_vol_10d"], "is_new": false}, {"model_id": "v2_h5_GLOBAL_XGBoost_N10", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 5, "n_features": 10, "F1_dir": 0.5081, "F1_UP_FORT": 0.2194, "F1_DOWN_FORT": 0.4213, "train_start": "2001-02-06", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["HON_Honeywell_zscore_60d__zrel__vix_vs_ma20", "VIX_Price_zscore_60d__div__NFCI_vol_20d", "vix_vs_ma10__zrel__US30Y_Rate_ret_20d", "PAYX_Paychex_vol_20d", "heston_var_ev_h7", "EWL_Switzerland_zscore_60d", "HON_Honeywell_zscore_60d__minus__HD_zscore_60d", "vix_zscore_10d__div__vix_vol_of_vol_10d", "heston_var_ev_h5", "AORD_AUS_zscore_60d"], "is_new": false}, {"model_id": "v2_h5_GLOBAL_XGBoost_N11", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 5, "n_features": 11, "F1_dir": 0.5134, "F1_UP_FORT": 0.2264, "F1_DOWN_FORT": 0.4244, "train_start": "2001-02-06", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["HON_Honeywell_zscore_60d__zrel__vix_vs_ma20", "VIX_Price_zscore_60d__div__NFCI_vol_20d", "vix_vs_ma10__zrel__US30Y_Rate_ret_20d", "PAYX_Paychex_vol_20d", "heston_var_ev_h7", "EWL_Switzerland_zscore_60d", "HON_Honeywell_zscore_60d__minus__HD_zscore_60d", "vix_zscore_10d__div__vix_vol_of_vol_10d", "heston_var_ev_h5", "AORD_AUS_zscore_60d", "VIX_Price_zscore_60d__minus__HON_Honeywell_zscore_60d"], "is_new": false}, {"model_id": "v2_h5_GLOBAL_XGBoost_N12", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 5, "n_features": 12, "F1_dir": 0.5152, "F1_UP_FORT": 0.2321, "F1_DOWN_FORT": 0.4205, "train_start": "2001-02-06", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["HON_Honeywell_zscore_60d__zrel__vix_vs_ma20", "VIX_Price_zscore_60d__div__NFCI_vol_20d", "vix_vs_ma10__zrel__US30Y_Rate_ret_20d", "PAYX_Paychex_vol_20d", "heston_var_ev_h7", "EWL_Switzerland_zscore_60d", "HON_Honeywell_zscore_60d__minus__HD_zscore_60d", "vix_zscore_10d__div__vix_vol_of_vol_10d", "heston_var_ev_h5", "AORD_AUS_zscore_60d", "VIX_Price_zscore_60d__minus__HON_Honeywell_zscore_60d", "AXP_Amex_vol_20d"], "is_new": false}, {"model_id": "v2_h5_GLOBAL_XGBoost_N13", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 5, "n_features": 13, "F1_dir": 0.5117, "F1_UP_FORT": 0.2678, "F1_DOWN_FORT": 0.4079, "train_start": "2001-02-06", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["HON_Honeywell_zscore_60d__zrel__vix_vs_ma20", "VIX_Price_zscore_60d__div__NFCI_vol_20d", "vix_vs_ma10__zrel__US30Y_Rate_ret_20d", "PAYX_Paychex_vol_20d", "heston_var_ev_h7", "EWL_Switzerland_zscore_60d", "HON_Honeywell_zscore_60d__minus__HD_zscore_60d", "vix_zscore_10d__div__vix_vol_of_vol_10d", "heston_var_ev_h5", "AORD_AUS_zscore_60d", "VIX_Price_zscore_60d__minus__HON_Honeywell_zscore_60d", "AXP_Amex_vol_20d", "NFCI_zscore_60d__div__HD_zscore_60d"], "is_new": false}, {"model_id": "v2_h5_GLOBAL_XGBoost_N14", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 5, "n_features": 14, "F1_dir": 0.5198, "F1_UP_FORT": 0.2709, "F1_DOWN_FORT": 0.4143, "train_start": "2001-02-06", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["HON_Honeywell_zscore_60d__zrel__vix_vs_ma20", "VIX_Price_zscore_60d__div__NFCI_vol_20d", "vix_vs_ma10__zrel__US30Y_Rate_ret_20d", "PAYX_Paychex_vol_20d", "heston_var_ev_h7", "EWL_Switzerland_zscore_60d", "HON_Honeywell_zscore_60d__minus__HD_zscore_60d", "vix_zscore_10d__div__vix_vol_of_vol_10d", "heston_var_ev_h5", "AORD_AUS_zscore_60d", "VIX_Price_zscore_60d__minus__HON_Honeywell_zscore_60d", "AXP_Amex_vol_20d", "NFCI_zscore_60d__div__HD_zscore_60d", "vix_vol_of_vol_10d__minus__NFCI_vol_20d"], "is_new": false}, {"model_id": "v2_h5_GLOBAL_XGBoost_N15", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 5, "n_features": 15, "F1_dir": 0.5353, "F1_UP_FORT": 0.3187, "F1_DOWN_FORT": 0.4278, "train_start": "2001-02-06", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["HON_Honeywell_zscore_60d__zrel__vix_vs_ma20", "VIX_Price_zscore_60d__div__NFCI_vol_20d", "vix_vs_ma10__zrel__US30Y_Rate_ret_20d", "PAYX_Paychex_vol_20d", "heston_var_ev_h7", "EWL_Switzerland_zscore_60d", "HON_Honeywell_zscore_60d__minus__HD_zscore_60d", "vix_zscore_10d__div__vix_vol_of_vol_10d", "heston_var_ev_h5", "AORD_AUS_zscore_60d", "VIX_Price_zscore_60d__minus__HON_Honeywell_zscore_60d", "AXP_Amex_vol_20d", "NFCI_zscore_60d__div__HD_zscore_60d", "vix_vol_of_vol_10d__minus__NFCI_vol_20d", "VIX_Price_zscore_60d__minus__NFCI_zscore_60d"], "is_new": false}, {"model_id": "v2_h5_GLOBAL_XGBoost_N16", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 5, "n_features": 16, "F1_dir": 0.5333, "F1_UP_FORT": 0.3181, "F1_DOWN_FORT": 0.4377, "train_start": "2001-02-06", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["HON_Honeywell_zscore_60d__zrel__vix_vs_ma20", "VIX_Price_zscore_60d__div__NFCI_vol_20d", "vix_vs_ma10__zrel__US30Y_Rate_ret_20d", "PAYX_Paychex_vol_20d", "heston_var_ev_h7", "EWL_Switzerland_zscore_60d", "HON_Honeywell_zscore_60d__minus__HD_zscore_60d", "vix_zscore_10d__div__vix_vol_of_vol_10d", "heston_var_ev_h5", "AORD_AUS_zscore_60d", "VIX_Price_zscore_60d__minus__HON_Honeywell_zscore_60d", "AXP_Amex_vol_20d", "NFCI_zscore_60d__div__HD_zscore_60d", "vix_vol_of_vol_10d__minus__NFCI_vol_20d", "VIX_Price_zscore_60d__minus__NFCI_zscore_60d", "SO_SouthernCo_ret_5d"], "is_new": false}, {"model_id": "v2_h5_GLOBAL_XGBoost_N17", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 5, "n_features": 17, "F1_dir": 0.5435, "F1_UP_FORT": 0.3351, "F1_DOWN_FORT": 0.4413, "train_start": "2001-02-06", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["HON_Honeywell_zscore_60d__zrel__vix_vs_ma20", "VIX_Price_zscore_60d__div__NFCI_vol_20d", "vix_vs_ma10__zrel__US30Y_Rate_ret_20d", "PAYX_Paychex_vol_20d", "heston_var_ev_h7", "EWL_Switzerland_zscore_60d", "HON_Honeywell_zscore_60d__minus__HD_zscore_60d", "vix_zscore_10d__div__vix_vol_of_vol_10d", "heston_var_ev_h5", "AORD_AUS_zscore_60d", "VIX_Price_zscore_60d__minus__HON_Honeywell_zscore_60d", "AXP_Amex_vol_20d", "NFCI_zscore_60d__div__HD_zscore_60d", "vix_vol_of_vol_10d__minus__NFCI_vol_20d", "VIX_Price_zscore_60d__minus__NFCI_zscore_60d", "SO_SouthernCo_ret_5d", "3M_vol_20d"], "is_new": false}, {"model_id": "v2_h5_GLOBAL_LightGBM_N5", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 5, "n_features": 5, "F1_dir": 0.5118, "F1_UP_FORT": 0.2054, "F1_DOWN_FORT": 0.3629, "train_start": "2001-02-06", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["HON_Honeywell_zscore_60d__zrel__vix_vs_ma20", "VIX_Price_zscore_60d__div__NFCI_vol_20d", "vix_vs_ma10__zrel__US30Y_Rate_ret_20d", "PAYX_Paychex_vol_20d", "heston_var_ev_h7"], "is_new": false}, {"model_id": "v2_h5_GLOBAL_LightGBM_N7", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 5, "n_features": 7, "F1_dir": 0.5084, "F1_UP_FORT": 0.2131, "F1_DOWN_FORT": 0.4049, "train_start": "2001-02-06", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["HON_Honeywell_zscore_60d__zrel__vix_vs_ma20", "VIX_Price_zscore_60d__div__NFCI_vol_20d", "vix_vs_ma10__zrel__US30Y_Rate_ret_20d", "PAYX_Paychex_vol_20d", "heston_var_ev_h7", "EWL_Switzerland_zscore_60d", "HON_Honeywell_zscore_60d__minus__HD_zscore_60d"], "is_new": false}, {"model_id": "v2_h5_GLOBAL_LightGBM_N8", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 5, "n_features": 8, "F1_dir": 0.5332, "F1_UP_FORT": 0.2204, "F1_DOWN_FORT": 0.3894, "train_start": "2001-02-06", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["HON_Honeywell_zscore_60d__zrel__vix_vs_ma20", "VIX_Price_zscore_60d__div__NFCI_vol_20d", "vix_vs_ma10__zrel__US30Y_Rate_ret_20d", "PAYX_Paychex_vol_20d", "heston_var_ev_h7", "EWL_Switzerland_zscore_60d", "HON_Honeywell_zscore_60d__minus__HD_zscore_60d", "vix_zscore_10d__div__vix_vol_of_vol_10d"], "is_new": false}, {"model_id": "v2_h5_GLOBAL_LightGBM_N9", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 5, "n_features": 9, "F1_dir": 0.5317, "F1_UP_FORT": 0.2193, "F1_DOWN_FORT": 0.4064, "train_start": "2001-02-06", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["HON_Honeywell_zscore_60d__zrel__vix_vs_ma20", "VIX_Price_zscore_60d__div__NFCI_vol_20d", "vix_vs_ma10__zrel__US30Y_Rate_ret_20d", "PAYX_Paychex_vol_20d", "heston_var_ev_h7", "EWL_Switzerland_zscore_60d", "HON_Honeywell_zscore_60d__minus__HD_zscore_60d", "vix_zscore_10d__div__vix_vol_of_vol_10d", "heston_var_ev_h5"], "is_new": false}, {"model_id": "v2_h5_GLOBAL_LightGBM_N10", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 5, "n_features": 10, "F1_dir": 0.5292, "F1_UP_FORT": 0.25, "F1_DOWN_FORT": 0.3874, "train_start": "2001-02-06", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["HON_Honeywell_zscore_60d__zrel__vix_vs_ma20", "VIX_Price_zscore_60d__div__NFCI_vol_20d", "vix_vs_ma10__zrel__US30Y_Rate_ret_20d", "PAYX_Paychex_vol_20d", "heston_var_ev_h7", "EWL_Switzerland_zscore_60d", "HON_Honeywell_zscore_60d__minus__HD_zscore_60d", "vix_zscore_10d__div__vix_vol_of_vol_10d", "heston_var_ev_h5", "AORD_AUS_zscore_60d"], "is_new": false}, {"model_id": "v2_h5_GLOBAL_LightGBM_N11", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 5, "n_features": 11, "F1_dir": 0.5105, "F1_UP_FORT": 0.2295, "F1_DOWN_FORT": 0.3955, "train_start": "2001-02-06", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["HON_Honeywell_zscore_60d__zrel__vix_vs_ma20", "VIX_Price_zscore_60d__div__NFCI_vol_20d", "vix_vs_ma10__zrel__US30Y_Rate_ret_20d", "PAYX_Paychex_vol_20d", "heston_var_ev_h7", "EWL_Switzerland_zscore_60d", "HON_Honeywell_zscore_60d__minus__HD_zscore_60d", "vix_zscore_10d__div__vix_vol_of_vol_10d", "heston_var_ev_h5", "AORD_AUS_zscore_60d", "VIX_Price_zscore_60d__minus__HON_Honeywell_zscore_60d"], "is_new": false}, {"model_id": "v2_h5_GLOBAL_LightGBM_N12", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 5, "n_features": 12, "F1_dir": 0.5147, "F1_UP_FORT": 0.2313, "F1_DOWN_FORT": 0.4183, "train_start": "2001-02-06", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["HON_Honeywell_zscore_60d__zrel__vix_vs_ma20", "VIX_Price_zscore_60d__div__NFCI_vol_20d", "vix_vs_ma10__zrel__US30Y_Rate_ret_20d", "PAYX_Paychex_vol_20d", "heston_var_ev_h7", "EWL_Switzerland_zscore_60d", "HON_Honeywell_zscore_60d__minus__HD_zscore_60d", "vix_zscore_10d__div__vix_vol_of_vol_10d", "heston_var_ev_h5", "AORD_AUS_zscore_60d", "VIX_Price_zscore_60d__minus__HON_Honeywell_zscore_60d", "AXP_Amex_vol_20d"], "is_new": false}, {"model_id": "v2_h5_GLOBAL_LightGBM_N13", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 5, "n_features": 13, "F1_dir": 0.5339, "F1_UP_FORT": 0.2544, "F1_DOWN_FORT": 0.4066, "train_start": "2001-02-06", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["HON_Honeywell_zscore_60d__zrel__vix_vs_ma20", "VIX_Price_zscore_60d__div__NFCI_vol_20d", "vix_vs_ma10__zrel__US30Y_Rate_ret_20d", "PAYX_Paychex_vol_20d", "heston_var_ev_h7", "EWL_Switzerland_zscore_60d", "HON_Honeywell_zscore_60d__minus__HD_zscore_60d", "vix_zscore_10d__div__vix_vol_of_vol_10d", "heston_var_ev_h5", "AORD_AUS_zscore_60d", "VIX_Price_zscore_60d__minus__HON_Honeywell_zscore_60d", "AXP_Amex_vol_20d", "NFCI_zscore_60d__div__HD_zscore_60d"], "is_new": false}, {"model_id": "v2_h5_GLOBAL_LightGBM_N14", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 5, "n_features": 14, "F1_dir": 0.512, "F1_UP_FORT": 0.2773, "F1_DOWN_FORT": 0.4032, "train_start": "2001-02-06", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["HON_Honeywell_zscore_60d__zrel__vix_vs_ma20", "VIX_Price_zscore_60d__div__NFCI_vol_20d", "vix_vs_ma10__zrel__US30Y_Rate_ret_20d", "PAYX_Paychex_vol_20d", "heston_var_ev_h7", "EWL_Switzerland_zscore_60d", "HON_Honeywell_zscore_60d__minus__HD_zscore_60d", "vix_zscore_10d__div__vix_vol_of_vol_10d", "heston_var_ev_h5", "AORD_AUS_zscore_60d", "VIX_Price_zscore_60d__minus__HON_Honeywell_zscore_60d", "AXP_Amex_vol_20d", "NFCI_zscore_60d__div__HD_zscore_60d", "vix_vol_of_vol_10d__minus__NFCI_vol_20d"], "is_new": false}, {"model_id": "v2_h5_GLOBAL_LightGBM_N15", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 5, "n_features": 15, "F1_dir": 0.5384, "F1_UP_FORT": 0.342, "F1_DOWN_FORT": 0.4152, "train_start": "2001-02-06", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["HON_Honeywell_zscore_60d__zrel__vix_vs_ma20", "VIX_Price_zscore_60d__div__NFCI_vol_20d", "vix_vs_ma10__zrel__US30Y_Rate_ret_20d", "PAYX_Paychex_vol_20d", "heston_var_ev_h7", "EWL_Switzerland_zscore_60d", "HON_Honeywell_zscore_60d__minus__HD_zscore_60d", "vix_zscore_10d__div__vix_vol_of_vol_10d", "heston_var_ev_h5", "AORD_AUS_zscore_60d", "VIX_Price_zscore_60d__minus__HON_Honeywell_zscore_60d", "AXP_Amex_vol_20d", "NFCI_zscore_60d__div__HD_zscore_60d", "vix_vol_of_vol_10d__minus__NFCI_vol_20d", "VIX_Price_zscore_60d__minus__NFCI_zscore_60d"], "is_new": false}, {"model_id": "v2_h5_GLOBAL_LightGBM_N16", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 5, "n_features": 16, "F1_dir": 0.5229, "F1_UP_FORT": 0.2963, "F1_DOWN_FORT": 0.4229, "train_start": "2001-02-06", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["HON_Honeywell_zscore_60d__zrel__vix_vs_ma20", "VIX_Price_zscore_60d__div__NFCI_vol_20d", "vix_vs_ma10__zrel__US30Y_Rate_ret_20d", "PAYX_Paychex_vol_20d", "heston_var_ev_h7", "EWL_Switzerland_zscore_60d", "HON_Honeywell_zscore_60d__minus__HD_zscore_60d", "vix_zscore_10d__div__vix_vol_of_vol_10d", "heston_var_ev_h5", "AORD_AUS_zscore_60d", "VIX_Price_zscore_60d__minus__HON_Honeywell_zscore_60d", "AXP_Amex_vol_20d", "NFCI_zscore_60d__div__HD_zscore_60d", "vix_vol_of_vol_10d__minus__NFCI_vol_20d", "VIX_Price_zscore_60d__minus__NFCI_zscore_60d", "SO_SouthernCo_ret_5d"], "is_new": false}, {"model_id": "v2_h5_GLOBAL_LightGBM_N17", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 5, "n_features": 17, "F1_dir": 0.5457, "F1_UP_FORT": 0.3311, "F1_DOWN_FORT": 0.4268, "train_start": "2001-02-06", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["HON_Honeywell_zscore_60d__zrel__vix_vs_ma20", "VIX_Price_zscore_60d__div__NFCI_vol_20d", "vix_vs_ma10__zrel__US30Y_Rate_ret_20d", "PAYX_Paychex_vol_20d", "heston_var_ev_h7", "EWL_Switzerland_zscore_60d", "HON_Honeywell_zscore_60d__minus__HD_zscore_60d", "vix_zscore_10d__div__vix_vol_of_vol_10d", "heston_var_ev_h5", "AORD_AUS_zscore_60d", "VIX_Price_zscore_60d__minus__HON_Honeywell_zscore_60d", "AXP_Amex_vol_20d", "NFCI_zscore_60d__div__HD_zscore_60d", "vix_vol_of_vol_10d__minus__NFCI_vol_20d", "VIX_Price_zscore_60d__minus__NFCI_zscore_60d", "SO_SouthernCo_ret_5d", "3M_vol_20d"], "is_new": false}, {"model_id": "v2_h5_GLOBAL_GradientBoosting_N5", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 5, "n_features": 5, "F1_dir": 0.5187, "F1_UP_FORT": 0.1932, "F1_DOWN_FORT": 0.4084, "train_start": "2001-02-06", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["HON_Honeywell_zscore_60d__zrel__vix_vs_ma20", "VIX_Price_zscore_60d__div__NFCI_vol_20d", "vix_vs_ma10__zrel__US30Y_Rate_ret_20d", "PAYX_Paychex_vol_20d", "heston_var_ev_h7"], "is_new": false}, {"model_id": "v2_h5_GLOBAL_GradientBoosting_N6", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 5, "n_features": 6, "F1_dir": 0.5017, "F1_UP_FORT": 0.2068, "F1_DOWN_FORT": 0.4134, "train_start": "2001-02-06", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["HON_Honeywell_zscore_60d__zrel__vix_vs_ma20", "VIX_Price_zscore_60d__div__NFCI_vol_20d", "vix_vs_ma10__zrel__US30Y_Rate_ret_20d", "PAYX_Paychex_vol_20d", "heston_var_ev_h7", "EWL_Switzerland_zscore_60d"], "is_new": false}, {"model_id": "v2_h5_GLOBAL_GradientBoosting_N7", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 5, "n_features": 7, "F1_dir": 0.5077, "F1_UP_FORT": 0.2159, "F1_DOWN_FORT": 0.403, "train_start": "2001-02-06", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["HON_Honeywell_zscore_60d__zrel__vix_vs_ma20", "VIX_Price_zscore_60d__div__NFCI_vol_20d", "vix_vs_ma10__zrel__US30Y_Rate_ret_20d", "PAYX_Paychex_vol_20d", "heston_var_ev_h7", "EWL_Switzerland_zscore_60d", "HON_Honeywell_zscore_60d__minus__HD_zscore_60d"], "is_new": false}, {"model_id": "v2_h5_GLOBAL_GradientBoosting_N8", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 5, "n_features": 8, "F1_dir": 0.5029, "F1_UP_FORT": 0.2017, "F1_DOWN_FORT": 0.4039, "train_start": "2001-02-06", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["HON_Honeywell_zscore_60d__zrel__vix_vs_ma20", "VIX_Price_zscore_60d__div__NFCI_vol_20d", "vix_vs_ma10__zrel__US30Y_Rate_ret_20d", "PAYX_Paychex_vol_20d", "heston_var_ev_h7", "EWL_Switzerland_zscore_60d", "HON_Honeywell_zscore_60d__minus__HD_zscore_60d", "vix_zscore_10d__div__vix_vol_of_vol_10d"], "is_new": false}, {"model_id": "v2_h5_GLOBAL_GradientBoosting_N11", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 5, "n_features": 11, "F1_dir": 0.5136, "F1_UP_FORT": 0.2156, "F1_DOWN_FORT": 0.4055, "train_start": "2001-02-06", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["HON_Honeywell_zscore_60d__zrel__vix_vs_ma20", "VIX_Price_zscore_60d__div__NFCI_vol_20d", "vix_vs_ma10__zrel__US30Y_Rate_ret_20d", "PAYX_Paychex_vol_20d", "heston_var_ev_h7", "EWL_Switzerland_zscore_60d", "HON_Honeywell_zscore_60d__minus__HD_zscore_60d", "vix_zscore_10d__div__vix_vol_of_vol_10d", "heston_var_ev_h5", "AORD_AUS_zscore_60d", "VIX_Price_zscore_60d__minus__HON_Honeywell_zscore_60d"], "is_new": false}, {"model_id": "v2_h5_GLOBAL_GradientBoosting_N12", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 5, "n_features": 12, "F1_dir": 0.5172, "F1_UP_FORT": 0.256, "F1_DOWN_FORT": 0.4106, "train_start": "2001-02-06", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["HON_Honeywell_zscore_60d__zrel__vix_vs_ma20", "VIX_Price_zscore_60d__div__NFCI_vol_20d", "vix_vs_ma10__zrel__US30Y_Rate_ret_20d", "PAYX_Paychex_vol_20d", "heston_var_ev_h7", "EWL_Switzerland_zscore_60d", "HON_Honeywell_zscore_60d__minus__HD_zscore_60d", "vix_zscore_10d__div__vix_vol_of_vol_10d", "heston_var_ev_h5", "AORD_AUS_zscore_60d", "VIX_Price_zscore_60d__minus__HON_Honeywell_zscore_60d", "AXP_Amex_vol_20d"], "is_new": false}, {"model_id": "v2_h5_GLOBAL_GradientBoosting_N13", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 5, "n_features": 13, "F1_dir": 0.5097, "F1_UP_FORT": 0.2561, "F1_DOWN_FORT": 0.4376, "train_start": "2001-02-06", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["HON_Honeywell_zscore_60d__zrel__vix_vs_ma20", "VIX_Price_zscore_60d__div__NFCI_vol_20d", "vix_vs_ma10__zrel__US30Y_Rate_ret_20d", "PAYX_Paychex_vol_20d", "heston_var_ev_h7", "EWL_Switzerland_zscore_60d", "HON_Honeywell_zscore_60d__minus__HD_zscore_60d", "vix_zscore_10d__div__vix_vol_of_vol_10d", "heston_var_ev_h5", "AORD_AUS_zscore_60d", "VIX_Price_zscore_60d__minus__HON_Honeywell_zscore_60d", "AXP_Amex_vol_20d", "NFCI_zscore_60d__div__HD_zscore_60d"], "is_new": false}, {"model_id": "v2_h5_GLOBAL_GradientBoosting_N14", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 5, "n_features": 14, "F1_dir": 0.5246, "F1_UP_FORT": 0.2809, "F1_DOWN_FORT": 0.4095, "train_start": "2001-02-06", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["HON_Honeywell_zscore_60d__zrel__vix_vs_ma20", "VIX_Price_zscore_60d__div__NFCI_vol_20d", "vix_vs_ma10__zrel__US30Y_Rate_ret_20d", "PAYX_Paychex_vol_20d", "heston_var_ev_h7", "EWL_Switzerland_zscore_60d", "HON_Honeywell_zscore_60d__minus__HD_zscore_60d", "vix_zscore_10d__div__vix_vol_of_vol_10d", "heston_var_ev_h5", "AORD_AUS_zscore_60d", "VIX_Price_zscore_60d__minus__HON_Honeywell_zscore_60d", "AXP_Amex_vol_20d", "NFCI_zscore_60d__div__HD_zscore_60d", "vix_vol_of_vol_10d__minus__NFCI_vol_20d"], "is_new": false}, {"model_id": "v2_h5_GLOBAL_GradientBoosting_N15", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 5, "n_features": 15, "F1_dir": 0.5263, "F1_UP_FORT": 0.3237, "F1_DOWN_FORT": 0.4433, "train_start": "2001-02-06", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["HON_Honeywell_zscore_60d__zrel__vix_vs_ma20", "VIX_Price_zscore_60d__div__NFCI_vol_20d", "vix_vs_ma10__zrel__US30Y_Rate_ret_20d", "PAYX_Paychex_vol_20d", "heston_var_ev_h7", "EWL_Switzerland_zscore_60d", "HON_Honeywell_zscore_60d__minus__HD_zscore_60d", "vix_zscore_10d__div__vix_vol_of_vol_10d", "heston_var_ev_h5", "AORD_AUS_zscore_60d", "VIX_Price_zscore_60d__minus__HON_Honeywell_zscore_60d", "AXP_Amex_vol_20d", "NFCI_zscore_60d__div__HD_zscore_60d", "vix_vol_of_vol_10d__minus__NFCI_vol_20d", "VIX_Price_zscore_60d__minus__NFCI_zscore_60d"], "is_new": false}, {"model_id": "v2_h5_GLOBAL_GradientBoosting_N16", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 5, "n_features": 16, "F1_dir": 0.5327, "F1_UP_FORT": 0.3397, "F1_DOWN_FORT": 0.4125, "train_start": "2001-02-06", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["HON_Honeywell_zscore_60d__zrel__vix_vs_ma20", "VIX_Price_zscore_60d__div__NFCI_vol_20d", "vix_vs_ma10__zrel__US30Y_Rate_ret_20d", "PAYX_Paychex_vol_20d", "heston_var_ev_h7", "EWL_Switzerland_zscore_60d", "HON_Honeywell_zscore_60d__minus__HD_zscore_60d", "vix_zscore_10d__div__vix_vol_of_vol_10d", "heston_var_ev_h5", "AORD_AUS_zscore_60d", "VIX_Price_zscore_60d__minus__HON_Honeywell_zscore_60d", "AXP_Amex_vol_20d", "NFCI_zscore_60d__div__HD_zscore_60d", "vix_vol_of_vol_10d__minus__NFCI_vol_20d", "VIX_Price_zscore_60d__minus__NFCI_zscore_60d", "SO_SouthernCo_ret_5d"], "is_new": false}, {"model_id": "v2_h5_GLOBAL_GradientBoosting_N17", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 5, "n_features": 17, "F1_dir": 0.5348, "F1_UP_FORT": 0.3432, "F1_DOWN_FORT": 0.4329, "train_start": "2001-02-06", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["HON_Honeywell_zscore_60d__zrel__vix_vs_ma20", "VIX_Price_zscore_60d__div__NFCI_vol_20d", "vix_vs_ma10__zrel__US30Y_Rate_ret_20d", "PAYX_Paychex_vol_20d", "heston_var_ev_h7", "EWL_Switzerland_zscore_60d", "HON_Honeywell_zscore_60d__minus__HD_zscore_60d", "vix_zscore_10d__div__vix_vol_of_vol_10d", "heston_var_ev_h5", "AORD_AUS_zscore_60d", "VIX_Price_zscore_60d__minus__HON_Honeywell_zscore_60d", "AXP_Amex_vol_20d", "NFCI_zscore_60d__div__HD_zscore_60d", "vix_vol_of_vol_10d__minus__NFCI_vol_20d", "VIX_Price_zscore_60d__minus__NFCI_zscore_60d", "SO_SouthernCo_ret_5d", "3M_vol_20d"], "is_new": false}, {"model_id": "v2_h5_GLOBAL_RandomForest_N5", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 5, "n_features": 5, "F1_dir": 0.5248, "F1_UP_FORT": 0.179, "F1_DOWN_FORT": 0.4208, "train_start": "2001-02-06", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["HON_Honeywell_zscore_60d__zrel__vix_vs_ma20", "VIX_Price_zscore_60d__div__NFCI_vol_20d", "vix_vs_ma10__zrel__US30Y_Rate_ret_20d", "PAYX_Paychex_vol_20d", "heston_var_ev_h7"], "is_new": false}, {"model_id": "v2_h5_GLOBAL_RandomForest_N6", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 5, "n_features": 6, "F1_dir": 0.5098, "F1_UP_FORT": 0.166, "F1_DOWN_FORT": 0.4435, "train_start": "2001-02-06", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["HON_Honeywell_zscore_60d__zrel__vix_vs_ma20", "VIX_Price_zscore_60d__div__NFCI_vol_20d", "vix_vs_ma10__zrel__US30Y_Rate_ret_20d", "PAYX_Paychex_vol_20d", "heston_var_ev_h7", "EWL_Switzerland_zscore_60d"], "is_new": false}, {"model_id": "v2_h5_GLOBAL_RandomForest_N9", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 5, "n_features": 9, "F1_dir": 0.5083, "F1_UP_FORT": 0.1382, "F1_DOWN_FORT": 0.4369, "train_start": "2001-02-06", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["HON_Honeywell_zscore_60d__zrel__vix_vs_ma20", "VIX_Price_zscore_60d__div__NFCI_vol_20d", "vix_vs_ma10__zrel__US30Y_Rate_ret_20d", "PAYX_Paychex_vol_20d", "heston_var_ev_h7", "EWL_Switzerland_zscore_60d", "HON_Honeywell_zscore_60d__minus__HD_zscore_60d", "vix_zscore_10d__div__vix_vol_of_vol_10d", "heston_var_ev_h5"], "is_new": false}, {"model_id": "v2_h5_GLOBAL_RandomForest_N11", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 5, "n_features": 11, "F1_dir": 0.5019, "F1_UP_FORT": 0.1552, "F1_DOWN_FORT": 0.4362, "train_start": "2001-02-06", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["HON_Honeywell_zscore_60d__zrel__vix_vs_ma20", "VIX_Price_zscore_60d__div__NFCI_vol_20d", "vix_vs_ma10__zrel__US30Y_Rate_ret_20d", "PAYX_Paychex_vol_20d", "heston_var_ev_h7", "EWL_Switzerland_zscore_60d", "HON_Honeywell_zscore_60d__minus__HD_zscore_60d", "vix_zscore_10d__div__vix_vol_of_vol_10d", "heston_var_ev_h5", "AORD_AUS_zscore_60d", "VIX_Price_zscore_60d__minus__HON_Honeywell_zscore_60d"], "is_new": false}, {"model_id": "v2_h5_GLOBAL_RandomForest_N12", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 5, "n_features": 12, "F1_dir": 0.5157, "F1_UP_FORT": 0.1511, "F1_DOWN_FORT": 0.4243, "train_start": "2001-02-06", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["HON_Honeywell_zscore_60d__zrel__vix_vs_ma20", "VIX_Price_zscore_60d__div__NFCI_vol_20d", "vix_vs_ma10__zrel__US30Y_Rate_ret_20d", "PAYX_Paychex_vol_20d", "heston_var_ev_h7", "EWL_Switzerland_zscore_60d", "HON_Honeywell_zscore_60d__minus__HD_zscore_60d", "vix_zscore_10d__div__vix_vol_of_vol_10d", "heston_var_ev_h5", "AORD_AUS_zscore_60d", "VIX_Price_zscore_60d__minus__HON_Honeywell_zscore_60d", "AXP_Amex_vol_20d"], "is_new": false}, {"model_id": "v2_h5_GLOBAL_RandomForest_N13", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 5, "n_features": 13, "F1_dir": 0.513, "F1_UP_FORT": 0.1591, "F1_DOWN_FORT": 0.4413, "train_start": "2001-02-06", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["HON_Honeywell_zscore_60d__zrel__vix_vs_ma20", "VIX_Price_zscore_60d__div__NFCI_vol_20d", "vix_vs_ma10__zrel__US30Y_Rate_ret_20d", "PAYX_Paychex_vol_20d", "heston_var_ev_h7", "EWL_Switzerland_zscore_60d", "HON_Honeywell_zscore_60d__minus__HD_zscore_60d", "vix_zscore_10d__div__vix_vol_of_vol_10d", "heston_var_ev_h5", "AORD_AUS_zscore_60d", "VIX_Price_zscore_60d__minus__HON_Honeywell_zscore_60d", "AXP_Amex_vol_20d", "NFCI_zscore_60d__div__HD_zscore_60d"], "is_new": false}, {"model_id": "v2_h5_GLOBAL_RandomForest_N14", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 5, "n_features": 14, "F1_dir": 0.5331, "F1_UP_FORT": 0.1905, "F1_DOWN_FORT": 0.4371, "train_start": "2001-02-06", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["HON_Honeywell_zscore_60d__zrel__vix_vs_ma20", "VIX_Price_zscore_60d__div__NFCI_vol_20d", "vix_vs_ma10__zrel__US30Y_Rate_ret_20d", "PAYX_Paychex_vol_20d", "heston_var_ev_h7", "EWL_Switzerland_zscore_60d", "HON_Honeywell_zscore_60d__minus__HD_zscore_60d", "vix_zscore_10d__div__vix_vol_of_vol_10d", "heston_var_ev_h5", "AORD_AUS_zscore_60d", "VIX_Price_zscore_60d__minus__HON_Honeywell_zscore_60d", "AXP_Amex_vol_20d", "NFCI_zscore_60d__div__HD_zscore_60d", "vix_vol_of_vol_10d__minus__NFCI_vol_20d"], "is_new": false}, {"model_id": "v2_h5_GLOBAL_RandomForest_N15", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 5, "n_features": 15, "F1_dir": 0.5375, "F1_UP_FORT": 0.2402, "F1_DOWN_FORT": 0.449, "train_start": "2001-02-06", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["HON_Honeywell_zscore_60d__zrel__vix_vs_ma20", "VIX_Price_zscore_60d__div__NFCI_vol_20d", "vix_vs_ma10__zrel__US30Y_Rate_ret_20d", "PAYX_Paychex_vol_20d", "heston_var_ev_h7", "EWL_Switzerland_zscore_60d", "HON_Honeywell_zscore_60d__minus__HD_zscore_60d", "vix_zscore_10d__div__vix_vol_of_vol_10d", "heston_var_ev_h5", "AORD_AUS_zscore_60d", "VIX_Price_zscore_60d__minus__HON_Honeywell_zscore_60d", "AXP_Amex_vol_20d", "NFCI_zscore_60d__div__HD_zscore_60d", "vix_vol_of_vol_10d__minus__NFCI_vol_20d", "VIX_Price_zscore_60d__minus__NFCI_zscore_60d"], "is_new": false}, {"model_id": "v2_h5_GLOBAL_RandomForest_N16", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 5, "n_features": 16, "F1_dir": 0.5304, "F1_UP_FORT": 0.276, "F1_DOWN_FORT": 0.4507, "train_start": "2001-02-06", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["HON_Honeywell_zscore_60d__zrel__vix_vs_ma20", "VIX_Price_zscore_60d__div__NFCI_vol_20d", "vix_vs_ma10__zrel__US30Y_Rate_ret_20d", "PAYX_Paychex_vol_20d", "heston_var_ev_h7", "EWL_Switzerland_zscore_60d", "HON_Honeywell_zscore_60d__minus__HD_zscore_60d", "vix_zscore_10d__div__vix_vol_of_vol_10d", "heston_var_ev_h5", "AORD_AUS_zscore_60d", "VIX_Price_zscore_60d__minus__HON_Honeywell_zscore_60d", "AXP_Amex_vol_20d", "NFCI_zscore_60d__div__HD_zscore_60d", "vix_vol_of_vol_10d__minus__NFCI_vol_20d", "VIX_Price_zscore_60d__minus__NFCI_zscore_60d", "SO_SouthernCo_ret_5d"], "is_new": false}, {"model_id": "v2_h5_GLOBAL_RandomForest_N17", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 5, "n_features": 17, "F1_dir": 0.5337, "F1_UP_FORT": 0.2716, "F1_DOWN_FORT": 0.4621, "train_start": "2001-02-06", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["HON_Honeywell_zscore_60d__zrel__vix_vs_ma20", "VIX_Price_zscore_60d__div__NFCI_vol_20d", "vix_vs_ma10__zrel__US30Y_Rate_ret_20d", "PAYX_Paychex_vol_20d", "heston_var_ev_h7", "EWL_Switzerland_zscore_60d", "HON_Honeywell_zscore_60d__minus__HD_zscore_60d", "vix_zscore_10d__div__vix_vol_of_vol_10d", "heston_var_ev_h5", "AORD_AUS_zscore_60d", "VIX_Price_zscore_60d__minus__HON_Honeywell_zscore_60d", "AXP_Amex_vol_20d", "NFCI_zscore_60d__div__HD_zscore_60d", "vix_vol_of_vol_10d__minus__NFCI_vol_20d", "VIX_Price_zscore_60d__minus__NFCI_zscore_60d", "SO_SouthernCo_ret_5d", "3M_vol_20d"], "is_new": false}, {"model_id": "v2_h5_GLOBAL_LogisticRegression_N5", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 5, "n_features": 5, "F1_dir": 0.604, "F1_UP_FORT": 0.3119, "F1_DOWN_FORT": 0.4545, "train_start": "2001-02-06", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["HON_Honeywell_zscore_60d__zrel__vix_vs_ma20", "VIX_Price_zscore_60d__div__NFCI_vol_20d", "vix_vs_ma10__zrel__US30Y_Rate_ret_20d", "PAYX_Paychex_vol_20d", "heston_var_ev_h7"], "is_new": false}, {"model_id": "v2_h5_GLOBAL_LogisticRegression_N6", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 5, "n_features": 6, "F1_dir": 0.5785, "F1_UP_FORT": 0.2878, "F1_DOWN_FORT": 0.4417, "train_start": "2001-02-06", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["HON_Honeywell_zscore_60d__zrel__vix_vs_ma20", "VIX_Price_zscore_60d__div__NFCI_vol_20d", "vix_vs_ma10__zrel__US30Y_Rate_ret_20d", "PAYX_Paychex_vol_20d", "heston_var_ev_h7", "EWL_Switzerland_zscore_60d"], "is_new": false}, {"model_id": "v2_h5_GLOBAL_LogisticRegression_N7", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 5, "n_features": 7, "F1_dir": 0.5696, "F1_UP_FORT": 0.2913, "F1_DOWN_FORT": 0.4507, "train_start": "2001-02-06", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["HON_Honeywell_zscore_60d__zrel__vix_vs_ma20", "VIX_Price_zscore_60d__div__NFCI_vol_20d", "vix_vs_ma10__zrel__US30Y_Rate_ret_20d", "PAYX_Paychex_vol_20d", "heston_var_ev_h7", "EWL_Switzerland_zscore_60d", "HON_Honeywell_zscore_60d__minus__HD_zscore_60d"], "is_new": false}, {"model_id": "v2_h5_GLOBAL_LogisticRegression_N8", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 5, "n_features": 8, "F1_dir": 0.5694, "F1_UP_FORT": 0.2907, "F1_DOWN_FORT": 0.4417, "train_start": "2001-02-06", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["HON_Honeywell_zscore_60d__zrel__vix_vs_ma20", "VIX_Price_zscore_60d__div__NFCI_vol_20d", "vix_vs_ma10__zrel__US30Y_Rate_ret_20d", "PAYX_Paychex_vol_20d", "heston_var_ev_h7", "EWL_Switzerland_zscore_60d", "HON_Honeywell_zscore_60d__minus__HD_zscore_60d", "vix_zscore_10d__div__vix_vol_of_vol_10d"], "is_new": false}, {"model_id": "v2_h5_GLOBAL_LogisticRegression_N9", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 5, "n_features": 9, "F1_dir": 0.5711, "F1_UP_FORT": 0.3122, "F1_DOWN_FORT": 0.4457, "train_start": "2001-02-06", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["HON_Honeywell_zscore_60d__zrel__vix_vs_ma20", "VIX_Price_zscore_60d__div__NFCI_vol_20d", "vix_vs_ma10__zrel__US30Y_Rate_ret_20d", "PAYX_Paychex_vol_20d", "heston_var_ev_h7", "EWL_Switzerland_zscore_60d", "HON_Honeywell_zscore_60d__minus__HD_zscore_60d", "vix_zscore_10d__div__vix_vol_of_vol_10d", "heston_var_ev_h5"], "is_new": false}, {"model_id": "v2_h5_GLOBAL_LogisticRegression_N10", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 5, "n_features": 10, "F1_dir": 0.5739, "F1_UP_FORT": 0.2653, "F1_DOWN_FORT": 0.4447, "train_start": "2001-02-06", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["HON_Honeywell_zscore_60d__zrel__vix_vs_ma20", "VIX_Price_zscore_60d__div__NFCI_vol_20d", "vix_vs_ma10__zrel__US30Y_Rate_ret_20d", "PAYX_Paychex_vol_20d", "heston_var_ev_h7", "EWL_Switzerland_zscore_60d", "HON_Honeywell_zscore_60d__minus__HD_zscore_60d", "vix_zscore_10d__div__vix_vol_of_vol_10d", "heston_var_ev_h5", "AORD_AUS_zscore_60d"], "is_new": false}, {"model_id": "v2_h5_GLOBAL_LogisticRegression_N11", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 5, "n_features": 11, "F1_dir": 0.5702, "F1_UP_FORT": 0.2599, "F1_DOWN_FORT": 0.4305, "train_start": "2001-02-06", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["HON_Honeywell_zscore_60d__zrel__vix_vs_ma20", "VIX_Price_zscore_60d__div__NFCI_vol_20d", "vix_vs_ma10__zrel__US30Y_Rate_ret_20d", "PAYX_Paychex_vol_20d", "heston_var_ev_h7", "EWL_Switzerland_zscore_60d", "HON_Honeywell_zscore_60d__minus__HD_zscore_60d", "vix_zscore_10d__div__vix_vol_of_vol_10d", "heston_var_ev_h5", "AORD_AUS_zscore_60d", "VIX_Price_zscore_60d__minus__HON_Honeywell_zscore_60d"], "is_new": false}, {"model_id": "v2_h5_GLOBAL_LogisticRegression_N12", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 5, "n_features": 12, "F1_dir": 0.5723, "F1_UP_FORT": 0.2569, "F1_DOWN_FORT": 0.4381, "train_start": "2001-02-06", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["HON_Honeywell_zscore_60d__zrel__vix_vs_ma20", "VIX_Price_zscore_60d__div__NFCI_vol_20d", "vix_vs_ma10__zrel__US30Y_Rate_ret_20d", "PAYX_Paychex_vol_20d", "heston_var_ev_h7", "EWL_Switzerland_zscore_60d", "HON_Honeywell_zscore_60d__minus__HD_zscore_60d", "vix_zscore_10d__div__vix_vol_of_vol_10d", "heston_var_ev_h5", "AORD_AUS_zscore_60d", "VIX_Price_zscore_60d__minus__HON_Honeywell_zscore_60d", "AXP_Amex_vol_20d"], "is_new": false}, {"model_id": "v2_h5_GLOBAL_LogisticRegression_N13", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 5, "n_features": 13, "F1_dir": 0.5706, "F1_UP_FORT": 0.2523, "F1_DOWN_FORT": 0.4388, "train_start": "2001-02-06", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["HON_Honeywell_zscore_60d__zrel__vix_vs_ma20", "VIX_Price_zscore_60d__div__NFCI_vol_20d", "vix_vs_ma10__zrel__US30Y_Rate_ret_20d", "PAYX_Paychex_vol_20d", "heston_var_ev_h7", "EWL_Switzerland_zscore_60d", "HON_Honeywell_zscore_60d__minus__HD_zscore_60d", "vix_zscore_10d__div__vix_vol_of_vol_10d", "heston_var_ev_h5", "AORD_AUS_zscore_60d", "VIX_Price_zscore_60d__minus__HON_Honeywell_zscore_60d", "AXP_Amex_vol_20d", "NFCI_zscore_60d__div__HD_zscore_60d"], "is_new": false}, {"model_id": "v2_h5_GLOBAL_LogisticRegression_N14", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 5, "n_features": 14, "F1_dir": 0.5831, "F1_UP_FORT": 0.2699, "F1_DOWN_FORT": 0.4424, "train_start": "2001-02-06", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["HON_Honeywell_zscore_60d__zrel__vix_vs_ma20", "VIX_Price_zscore_60d__div__NFCI_vol_20d", "vix_vs_ma10__zrel__US30Y_Rate_ret_20d", "PAYX_Paychex_vol_20d", "heston_var_ev_h7", "EWL_Switzerland_zscore_60d", "HON_Honeywell_zscore_60d__minus__HD_zscore_60d", "vix_zscore_10d__div__vix_vol_of_vol_10d", "heston_var_ev_h5", "AORD_AUS_zscore_60d", "VIX_Price_zscore_60d__minus__HON_Honeywell_zscore_60d", "AXP_Amex_vol_20d", "NFCI_zscore_60d__div__HD_zscore_60d", "vix_vol_of_vol_10d__minus__NFCI_vol_20d"], "is_new": false}, {"model_id": "v2_h5_GLOBAL_LogisticRegression_N15", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 5, "n_features": 15, "F1_dir": 0.564, "F1_UP_FORT": 0.3299, "F1_DOWN_FORT": 0.4569, "train_start": "2001-02-06", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["HON_Honeywell_zscore_60d__zrel__vix_vs_ma20", "VIX_Price_zscore_60d__div__NFCI_vol_20d", "vix_vs_ma10__zrel__US30Y_Rate_ret_20d", "PAYX_Paychex_vol_20d", "heston_var_ev_h7", "EWL_Switzerland_zscore_60d", "HON_Honeywell_zscore_60d__minus__HD_zscore_60d", "vix_zscore_10d__div__vix_vol_of_vol_10d", "heston_var_ev_h5", "AORD_AUS_zscore_60d", "VIX_Price_zscore_60d__minus__HON_Honeywell_zscore_60d", "AXP_Amex_vol_20d", "NFCI_zscore_60d__div__HD_zscore_60d", "vix_vol_of_vol_10d__minus__NFCI_vol_20d", "VIX_Price_zscore_60d__minus__NFCI_zscore_60d"], "is_new": false}, {"model_id": "v2_h5_GLOBAL_LogisticRegression_N16", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 5, "n_features": 16, "F1_dir": 0.5539, "F1_UP_FORT": 0.3128, "F1_DOWN_FORT": 0.4565, "train_start": "2001-02-06", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["HON_Honeywell_zscore_60d__zrel__vix_vs_ma20", "VIX_Price_zscore_60d__div__NFCI_vol_20d", "vix_vs_ma10__zrel__US30Y_Rate_ret_20d", "PAYX_Paychex_vol_20d", "heston_var_ev_h7", "EWL_Switzerland_zscore_60d", "HON_Honeywell_zscore_60d__minus__HD_zscore_60d", "vix_zscore_10d__div__vix_vol_of_vol_10d", "heston_var_ev_h5", "AORD_AUS_zscore_60d", "VIX_Price_zscore_60d__minus__HON_Honeywell_zscore_60d", "AXP_Amex_vol_20d", "NFCI_zscore_60d__div__HD_zscore_60d", "vix_vol_of_vol_10d__minus__NFCI_vol_20d", "VIX_Price_zscore_60d__minus__NFCI_zscore_60d", "SO_SouthernCo_ret_5d"], "is_new": false}, {"model_id": "v2_h5_GLOBAL_LogisticRegression_N17", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 5, "n_features": 17, "F1_dir": 0.5414, "F1_UP_FORT": 0.3313, "F1_DOWN_FORT": 0.4582, "train_start": "2001-02-06", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["HON_Honeywell_zscore_60d__zrel__vix_vs_ma20", "VIX_Price_zscore_60d__div__NFCI_vol_20d", "vix_vs_ma10__zrel__US30Y_Rate_ret_20d", "PAYX_Paychex_vol_20d", "heston_var_ev_h7", "EWL_Switzerland_zscore_60d", "HON_Honeywell_zscore_60d__minus__HD_zscore_60d", "vix_zscore_10d__div__vix_vol_of_vol_10d", "heston_var_ev_h5", "AORD_AUS_zscore_60d", "VIX_Price_zscore_60d__minus__HON_Honeywell_zscore_60d", "AXP_Amex_vol_20d", "NFCI_zscore_60d__div__HD_zscore_60d", "vix_vol_of_vol_10d__minus__NFCI_vol_20d", "VIX_Price_zscore_60d__minus__NFCI_zscore_60d", "SO_SouthernCo_ret_5d", "3M_vol_20d"], "is_new": false}, {"model_id": "v2_h5_GLOBAL_LogisticRegression_Optuna_N5", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 5, "n_features": 5, "F1_dir": 0.6017, "F1_UP_FORT": 0.3087, "F1_DOWN_FORT": 0.4536, "train_start": "2001-02-06", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["HON_Honeywell_zscore_60d__zrel__vix_vs_ma20", "VIX_Price_zscore_60d__div__NFCI_vol_20d", "vix_vs_ma10__zrel__US30Y_Rate_ret_20d", "PAYX_Paychex_vol_20d", "heston_var_ev_h7"], "is_new": false}, {"model_id": "v2_h5_GLOBAL_LogisticRegression_OptunaCal_N5", "algo": "LogisticRegressionCal", "regime": "GLOBAL", "horizon": 5, "n_features": 5, "F1_dir": 0.5518, "F1_UP_FORT": 0.1422, "F1_DOWN_FORT": 0.4489, "train_start": "2001-02-06", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["HON_Honeywell_zscore_60d__zrel__vix_vs_ma20", "VIX_Price_zscore_60d__div__NFCI_vol_20d", "vix_vs_ma10__zrel__US30Y_Rate_ret_20d", "PAYX_Paychex_vol_20d", "heston_var_ev_h7"], "is_new": false}, {"model_id": "v2_h7_CALM_XGBoost_N5", "algo": "XGBoost", "regime": "CALM", "horizon": 7, "n_features": 5, "F1_dir": 0.5885, "F1_UP_FORT": 0.359, "F1_DOWN_FORT": 0.4681, "train_start": "2001-02-19", "sampler": "SMOTE", "best_params": "{}", "features": ["NFCI_ret_5d__div__NVDA_vol_20d", "QCOM_ret_20d__zrel__PCE_zscore_60d", "vix_vs_ma20__zrel__XLK_Tech_zscore_60d", "spx_abs_ret_max_5d__ret5x__PCE_zscore_60d", "VIX_Price_zscore_60d__div__PCE_zscore_60d"], "is_new": false}, {"model_id": "v2_h7_CALM_XGBoost_N6", "algo": "XGBoost", "regime": "CALM", "horizon": 7, "n_features": 6, "F1_dir": 0.6037, "F1_UP_FORT": 0.3043, "F1_DOWN_FORT": 0.4138, "train_start": "2001-02-19", "sampler": "SMOTE", "best_params": "{}", "features": ["NFCI_ret_5d__div__NVDA_vol_20d", "QCOM_ret_20d__zrel__PCE_zscore_60d", "vix_vs_ma20__zrel__XLK_Tech_zscore_60d", "spx_abs_ret_max_5d__ret5x__PCE_zscore_60d", "VIX_Price_zscore_60d__div__PCE_zscore_60d", "DAX_Germany_zscore_60d"], "is_new": false}, {"model_id": "v2_h7_CALM_XGBoost_N7", "algo": "XGBoost", "regime": "CALM", "horizon": 7, "n_features": 7, "F1_dir": 0.6178, "F1_UP_FORT": 0.3269, "F1_DOWN_FORT": 0.3778, "train_start": "2001-02-19", "sampler": "SMOTE", "best_params": "{}", "features": ["NFCI_ret_5d__div__NVDA_vol_20d", "QCOM_ret_20d__zrel__PCE_zscore_60d", "vix_vs_ma20__zrel__XLK_Tech_zscore_60d", "spx_abs_ret_max_5d__ret5x__PCE_zscore_60d", "VIX_Price_zscore_60d__div__PCE_zscore_60d", "DAX_Germany_zscore_60d", "LOW_Lowes_ret_5d"], "is_new": false}, {"model_id": "v2_h7_CALM_XGBoost_N8", "algo": "XGBoost", "regime": "CALM", "horizon": 7, "n_features": 8, "F1_dir": 0.5895, "F1_UP_FORT": 0.3107, "F1_DOWN_FORT": 0.3789, "train_start": "2001-02-19", "sampler": "SMOTE", "best_params": "{}", "features": ["NFCI_ret_5d__div__NVDA_vol_20d", "QCOM_ret_20d__zrel__PCE_zscore_60d", "vix_vs_ma20__zrel__XLK_Tech_zscore_60d", "spx_abs_ret_max_5d__ret5x__PCE_zscore_60d", "VIX_Price_zscore_60d__div__PCE_zscore_60d", "DAX_Germany_zscore_60d", "LOW_Lowes_ret_5d", "CPB_CampbellSoup_vol_20d"], "is_new": false}, {"model_id": "v2_h7_CALM_XGBoost_N9", "algo": "XGBoost", "regime": "CALM", "horizon": 7, "n_features": 9, "F1_dir": 0.5977, "F1_UP_FORT": 0.2889, "F1_DOWN_FORT": 0.3636, "train_start": "2001-02-19", "sampler": "SMOTE", "best_params": "{}", "features": ["NFCI_ret_5d__div__NVDA_vol_20d", "QCOM_ret_20d__zrel__PCE_zscore_60d", "vix_vs_ma20__zrel__XLK_Tech_zscore_60d", "spx_abs_ret_max_5d__ret5x__PCE_zscore_60d", "VIX_Price_zscore_60d__div__PCE_zscore_60d", "DAX_Germany_zscore_60d", "LOW_Lowes_ret_5d", "CPB_CampbellSoup_vol_20d", "CI_Cigna_vol_20d"], "is_new": false}, {"model_id": "v2_h7_CALM_XGBoost_N10", "algo": "XGBoost", "regime": "CALM", "horizon": 7, "n_features": 10, "F1_dir": 0.5675, "F1_UP_FORT": 0.3226, "F1_DOWN_FORT": 0.3636, "train_start": "2001-02-19", "sampler": "SMOTE", "best_params": "{}", "features": ["NFCI_ret_5d__div__NVDA_vol_20d", "QCOM_ret_20d__zrel__PCE_zscore_60d", "vix_vs_ma20__zrel__XLK_Tech_zscore_60d", "spx_abs_ret_max_5d__ret5x__PCE_zscore_60d", "VIX_Price_zscore_60d__div__PCE_zscore_60d", "DAX_Germany_zscore_60d", "LOW_Lowes_ret_5d", "CPB_CampbellSoup_vol_20d", "CI_Cigna_vol_20d", "vix_vs_ma20__prod__NVDA_vol_20d"], "is_new": false}, {"model_id": "v2_h7_CALM_XGBoost_N11", "algo": "XGBoost", "regime": "CALM", "horizon": 7, "n_features": 11, "F1_dir": 0.6101, "F1_UP_FORT": 0.3333, "F1_DOWN_FORT": 0.354, "train_start": "2001-02-19", "sampler": "SMOTE", "best_params": "{}", "features": ["NFCI_ret_5d__div__NVDA_vol_20d", "QCOM_ret_20d__zrel__PCE_zscore_60d", "vix_vs_ma20__zrel__XLK_Tech_zscore_60d", "spx_abs_ret_max_5d__ret5x__PCE_zscore_60d", "VIX_Price_zscore_60d__div__PCE_zscore_60d", "DAX_Germany_zscore_60d", "LOW_Lowes_ret_5d", "CPB_CampbellSoup_vol_20d", "CI_Cigna_vol_20d", "vix_vs_ma20__prod__NVDA_vol_20d", "VOD_Vodafone_zscore_60d"], "is_new": false}, {"model_id": "v2_h7_CALM_XGBoost_N12", "algo": "XGBoost", "regime": "CALM", "horizon": 7, "n_features": 12, "F1_dir": 0.5856, "F1_UP_FORT": 0.2826, "F1_DOWN_FORT": 0.3419, "train_start": "2001-02-19", "sampler": "SMOTE", "best_params": "{}", "features": ["NFCI_ret_5d__div__NVDA_vol_20d", "QCOM_ret_20d__zrel__PCE_zscore_60d", "vix_vs_ma20__zrel__XLK_Tech_zscore_60d", "spx_abs_ret_max_5d__ret5x__PCE_zscore_60d", "VIX_Price_zscore_60d__div__PCE_zscore_60d", "DAX_Germany_zscore_60d", "LOW_Lowes_ret_5d", "CPB_CampbellSoup_vol_20d", "CI_Cigna_vol_20d", "vix_vs_ma20__prod__NVDA_vol_20d", "VOD_Vodafone_zscore_60d", "BA_ret_1d"], "is_new": false}, {"model_id": "v2_h7_CALM_LightGBM_N5", "algo": "LightGBM", "regime": "CALM", "horizon": 7, "n_features": 5, "F1_dir": 0.5713, "F1_UP_FORT": 0.3636, "F1_DOWN_FORT": 0.4421, "train_start": "2001-02-19", "sampler": "SMOTE", "best_params": "{}", "features": ["NFCI_ret_5d__div__NVDA_vol_20d", "QCOM_ret_20d__zrel__PCE_zscore_60d", "vix_vs_ma20__zrel__XLK_Tech_zscore_60d", "spx_abs_ret_max_5d__ret5x__PCE_zscore_60d", "VIX_Price_zscore_60d__div__PCE_zscore_60d"], "is_new": false}, {"model_id": "v2_h7_CALM_LightGBM_N6", "algo": "LightGBM", "regime": "CALM", "horizon": 7, "n_features": 6, "F1_dir": 0.6396, "F1_UP_FORT": 0.3299, "F1_DOWN_FORT": 0.3902, "train_start": "2001-02-19", "sampler": "SMOTE", "best_params": "{}", "features": ["NFCI_ret_5d__div__NVDA_vol_20d", "QCOM_ret_20d__zrel__PCE_zscore_60d", "vix_vs_ma20__zrel__XLK_Tech_zscore_60d", "spx_abs_ret_max_5d__ret5x__PCE_zscore_60d", "VIX_Price_zscore_60d__div__PCE_zscore_60d", "DAX_Germany_zscore_60d"], "is_new": false}, {"model_id": "v2_h7_CALM_LightGBM_N7", "algo": "LightGBM", "regime": "CALM", "horizon": 7, "n_features": 7, "F1_dir": 0.5973, "F1_UP_FORT": 0.3148, "F1_DOWN_FORT": 0.3265, "train_start": "2001-02-19", "sampler": "SMOTE", "best_params": "{}", "features": ["NFCI_ret_5d__div__NVDA_vol_20d", "QCOM_ret_20d__zrel__PCE_zscore_60d", "vix_vs_ma20__zrel__XLK_Tech_zscore_60d", "spx_abs_ret_max_5d__ret5x__PCE_zscore_60d", "VIX_Price_zscore_60d__div__PCE_zscore_60d", "DAX_Germany_zscore_60d", "LOW_Lowes_ret_5d"], "is_new": false}, {"model_id": "v2_h7_CALM_LightGBM_N8", "algo": "LightGBM", "regime": "CALM", "horizon": 7, "n_features": 8, "F1_dir": 0.5885, "F1_UP_FORT": 0.3077, "F1_DOWN_FORT": 0.3478, "train_start": "2001-02-19", "sampler": "SMOTE", "best_params": "{}", "features": ["NFCI_ret_5d__div__NVDA_vol_20d", "QCOM_ret_20d__zrel__PCE_zscore_60d", "vix_vs_ma20__zrel__XLK_Tech_zscore_60d", "spx_abs_ret_max_5d__ret5x__PCE_zscore_60d", "VIX_Price_zscore_60d__div__PCE_zscore_60d", "DAX_Germany_zscore_60d", "LOW_Lowes_ret_5d", "CPB_CampbellSoup_vol_20d"], "is_new": false}, {"model_id": "v2_h7_CALM_LightGBM_N9", "algo": "LightGBM", "regime": "CALM", "horizon": 7, "n_features": 9, "F1_dir": 0.6121, "F1_UP_FORT": 0.2857, "F1_DOWN_FORT": 0.419, "train_start": "2001-02-19", "sampler": "SMOTE", "best_params": "{}", "features": ["NFCI_ret_5d__div__NVDA_vol_20d", "QCOM_ret_20d__zrel__PCE_zscore_60d", "vix_vs_ma20__zrel__XLK_Tech_zscore_60d", "spx_abs_ret_max_5d__ret5x__PCE_zscore_60d", "VIX_Price_zscore_60d__div__PCE_zscore_60d", "DAX_Germany_zscore_60d", "LOW_Lowes_ret_5d", "CPB_CampbellSoup_vol_20d", "CI_Cigna_vol_20d"], "is_new": false}, {"model_id": "v2_h7_CALM_LightGBM_N10", "algo": "LightGBM", "regime": "CALM", "horizon": 7, "n_features": 10, "F1_dir": 0.6121, "F1_UP_FORT": 0.3077, "F1_DOWN_FORT": 0.3551, "train_start": "2001-02-19", "sampler": "SMOTE", "best_params": "{}", "features": ["NFCI_ret_5d__div__NVDA_vol_20d", "QCOM_ret_20d__zrel__PCE_zscore_60d", "vix_vs_ma20__zrel__XLK_Tech_zscore_60d", "spx_abs_ret_max_5d__ret5x__PCE_zscore_60d", "VIX_Price_zscore_60d__div__PCE_zscore_60d", "DAX_Germany_zscore_60d", "LOW_Lowes_ret_5d", "CPB_CampbellSoup_vol_20d", "CI_Cigna_vol_20d", "vix_vs_ma20__prod__NVDA_vol_20d"], "is_new": false}, {"model_id": "v2_h7_CALM_LightGBM_N11", "algo": "LightGBM", "regime": "CALM", "horizon": 7, "n_features": 11, "F1_dir": 0.5888, "F1_UP_FORT": 0.3019, "F1_DOWN_FORT": 0.3636, "train_start": "2001-02-19", "sampler": "SMOTE", "best_params": "{}", "features": ["NFCI_ret_5d__div__NVDA_vol_20d", "QCOM_ret_20d__zrel__PCE_zscore_60d", "vix_vs_ma20__zrel__XLK_Tech_zscore_60d", "spx_abs_ret_max_5d__ret5x__PCE_zscore_60d", "VIX_Price_zscore_60d__div__PCE_zscore_60d", "DAX_Germany_zscore_60d", "LOW_Lowes_ret_5d", "CPB_CampbellSoup_vol_20d", "CI_Cigna_vol_20d", "vix_vs_ma20__prod__NVDA_vol_20d", "VOD_Vodafone_zscore_60d"], "is_new": false}, {"model_id": "v2_h7_CALM_LightGBM_N12", "algo": "LightGBM", "regime": "CALM", "horizon": 7, "n_features": 12, "F1_dir": 0.605, "F1_UP_FORT": 0.3011, "F1_DOWN_FORT": 0.3717, "train_start": "2001-02-19", "sampler": "SMOTE", "best_params": "{}", "features": ["NFCI_ret_5d__div__NVDA_vol_20d", "QCOM_ret_20d__zrel__PCE_zscore_60d", "vix_vs_ma20__zrel__XLK_Tech_zscore_60d", "spx_abs_ret_max_5d__ret5x__PCE_zscore_60d", "VIX_Price_zscore_60d__div__PCE_zscore_60d", "DAX_Germany_zscore_60d", "LOW_Lowes_ret_5d", "CPB_CampbellSoup_vol_20d", "CI_Cigna_vol_20d", "vix_vs_ma20__prod__NVDA_vol_20d", "VOD_Vodafone_zscore_60d", "BA_ret_1d"], "is_new": false}, {"model_id": "v2_h7_CALM_GradientBoosting_N5", "algo": "GradientBoosting", "regime": "CALM", "horizon": 7, "n_features": 5, "F1_dir": 0.5541, "F1_UP_FORT": 0.3291, "F1_DOWN_FORT": 0.4468, "train_start": "2001-02-19", "sampler": "SMOTE", "best_params": "{}", "features": ["NFCI_ret_5d__div__NVDA_vol_20d", "QCOM_ret_20d__zrel__PCE_zscore_60d", "vix_vs_ma20__zrel__XLK_Tech_zscore_60d", "spx_abs_ret_max_5d__ret5x__PCE_zscore_60d", "VIX_Price_zscore_60d__div__PCE_zscore_60d"], "is_new": false}, {"model_id": "v2_h7_CALM_GradientBoosting_N6", "algo": "GradientBoosting", "regime": "CALM", "horizon": 7, "n_features": 6, "F1_dir": 0.6083, "F1_UP_FORT": 0.2955, "F1_DOWN_FORT": 0.3871, "train_start": "2001-02-19", "sampler": "SMOTE", "best_params": "{}", "features": ["NFCI_ret_5d__div__NVDA_vol_20d", "QCOM_ret_20d__zrel__PCE_zscore_60d", "vix_vs_ma20__zrel__XLK_Tech_zscore_60d", "spx_abs_ret_max_5d__ret5x__PCE_zscore_60d", "VIX_Price_zscore_60d__div__PCE_zscore_60d", "DAX_Germany_zscore_60d"], "is_new": false}, {"model_id": "v2_h7_CALM_GradientBoosting_N7", "algo": "GradientBoosting", "regime": "CALM", "horizon": 7, "n_features": 7, "F1_dir": 0.5699, "F1_UP_FORT": 0.2826, "F1_DOWN_FORT": 0.2796, "train_start": "2001-02-19", "sampler": "SMOTE", "best_params": "{}", "features": ["NFCI_ret_5d__div__NVDA_vol_20d", "QCOM_ret_20d__zrel__PCE_zscore_60d", "vix_vs_ma20__zrel__XLK_Tech_zscore_60d", "spx_abs_ret_max_5d__ret5x__PCE_zscore_60d", "VIX_Price_zscore_60d__div__PCE_zscore_60d", "DAX_Germany_zscore_60d", "LOW_Lowes_ret_5d"], "is_new": false}, {"model_id": "v2_h7_CALM_GradientBoosting_N8", "algo": "GradientBoosting", "regime": "CALM", "horizon": 7, "n_features": 8, "F1_dir": 0.5973, "F1_UP_FORT": 0.2737, "F1_DOWN_FORT": 0.2887, "train_start": "2001-02-19", "sampler": "SMOTE", "best_params": "{}", "features": ["NFCI_ret_5d__div__NVDA_vol_20d", "QCOM_ret_20d__zrel__PCE_zscore_60d", "vix_vs_ma20__zrel__XLK_Tech_zscore_60d", "spx_abs_ret_max_5d__ret5x__PCE_zscore_60d", "VIX_Price_zscore_60d__div__PCE_zscore_60d", "DAX_Germany_zscore_60d", "LOW_Lowes_ret_5d", "CPB_CampbellSoup_vol_20d"], "is_new": false}, {"model_id": "v2_h7_CALM_GradientBoosting_N9", "algo": "GradientBoosting", "regime": "CALM", "horizon": 7, "n_features": 9, "F1_dir": 0.5708, "F1_UP_FORT": 0.2766, "F1_DOWN_FORT": 0.3043, "train_start": "2001-02-19", "sampler": "SMOTE", "best_params": "{}", "features": ["NFCI_ret_5d__div__NVDA_vol_20d", "QCOM_ret_20d__zrel__PCE_zscore_60d", "vix_vs_ma20__zrel__XLK_Tech_zscore_60d", "spx_abs_ret_max_5d__ret5x__PCE_zscore_60d", "VIX_Price_zscore_60d__div__PCE_zscore_60d", "DAX_Germany_zscore_60d", "LOW_Lowes_ret_5d", "CPB_CampbellSoup_vol_20d", "CI_Cigna_vol_20d"], "is_new": false}, {"model_id": "v2_h7_CALM_GradientBoosting_N10", "algo": "GradientBoosting", "regime": "CALM", "horizon": 7, "n_features": 10, "F1_dir": 0.5787, "F1_UP_FORT": 0.2444, "F1_DOWN_FORT": 0.3636, "train_start": "2001-02-19", "sampler": "SMOTE", "best_params": "{}", "features": ["NFCI_ret_5d__div__NVDA_vol_20d", "QCOM_ret_20d__zrel__PCE_zscore_60d", "vix_vs_ma20__zrel__XLK_Tech_zscore_60d", "spx_abs_ret_max_5d__ret5x__PCE_zscore_60d", "VIX_Price_zscore_60d__div__PCE_zscore_60d", "DAX_Germany_zscore_60d", "LOW_Lowes_ret_5d", "CPB_CampbellSoup_vol_20d", "CI_Cigna_vol_20d", "vix_vs_ma20__prod__NVDA_vol_20d"], "is_new": false}, {"model_id": "v2_h7_CALM_GradientBoosting_N11", "algo": "GradientBoosting", "regime": "CALM", "horizon": 7, "n_features": 11, "F1_dir": 0.5817, "F1_UP_FORT": 0.2247, "F1_DOWN_FORT": 0.3252, "train_start": "2001-02-19", "sampler": "SMOTE", "best_params": "{}", "features": ["NFCI_ret_5d__div__NVDA_vol_20d", "QCOM_ret_20d__zrel__PCE_zscore_60d", "vix_vs_ma20__zrel__XLK_Tech_zscore_60d", "spx_abs_ret_max_5d__ret5x__PCE_zscore_60d", "VIX_Price_zscore_60d__div__PCE_zscore_60d", "DAX_Germany_zscore_60d", "LOW_Lowes_ret_5d", "CPB_CampbellSoup_vol_20d", "CI_Cigna_vol_20d", "vix_vs_ma20__prod__NVDA_vol_20d", "VOD_Vodafone_zscore_60d"], "is_new": false}, {"model_id": "v2_h7_CALM_GradientBoosting_N12", "algo": "GradientBoosting", "regime": "CALM", "horizon": 7, "n_features": 12, "F1_dir": 0.6209, "F1_UP_FORT": 0.2526, "F1_DOWN_FORT": 0.3306, "train_start": "2001-02-19", "sampler": "SMOTE", "best_params": "{}", "features": ["NFCI_ret_5d__div__NVDA_vol_20d", "QCOM_ret_20d__zrel__PCE_zscore_60d", "vix_vs_ma20__zrel__XLK_Tech_zscore_60d", "spx_abs_ret_max_5d__ret5x__PCE_zscore_60d", "VIX_Price_zscore_60d__div__PCE_zscore_60d", "DAX_Germany_zscore_60d", "LOW_Lowes_ret_5d", "CPB_CampbellSoup_vol_20d", "CI_Cigna_vol_20d", "vix_vs_ma20__prod__NVDA_vol_20d", "VOD_Vodafone_zscore_60d", "BA_ret_1d"], "is_new": false}, {"model_id": "v2_h7_CALM_RandomForest_N5", "algo": "RandomForest", "regime": "CALM", "horizon": 7, "n_features": 5, "F1_dir": 0.5838, "F1_UP_FORT": 0.3488, "F1_DOWN_FORT": 0.4632, "train_start": "2001-02-19", "sampler": "SMOTE", "best_params": "{}", "features": ["NFCI_ret_5d__div__NVDA_vol_20d", "QCOM_ret_20d__zrel__PCE_zscore_60d", "vix_vs_ma20__zrel__XLK_Tech_zscore_60d", "spx_abs_ret_max_5d__ret5x__PCE_zscore_60d", "VIX_Price_zscore_60d__div__PCE_zscore_60d"], "is_new": false}, {"model_id": "v2_h7_CALM_RandomForest_N6", "algo": "RandomForest", "regime": "CALM", "horizon": 7, "n_features": 6, "F1_dir": 0.5627, "F1_UP_FORT": 0.3469, "F1_DOWN_FORT": 0.3736, "train_start": "2001-02-19", "sampler": "SMOTE", "best_params": "{}", "features": ["NFCI_ret_5d__div__NVDA_vol_20d", "QCOM_ret_20d__zrel__PCE_zscore_60d", "vix_vs_ma20__zrel__XLK_Tech_zscore_60d", "spx_abs_ret_max_5d__ret5x__PCE_zscore_60d", "VIX_Price_zscore_60d__div__PCE_zscore_60d", "DAX_Germany_zscore_60d"], "is_new": false}, {"model_id": "v2_h7_CALM_RandomForest_N7", "algo": "RandomForest", "regime": "CALM", "horizon": 7, "n_features": 7, "F1_dir": 0.5661, "F1_UP_FORT": 0.2857, "F1_DOWN_FORT": 0.3656, "train_start": "2001-02-19", "sampler": "SMOTE", "best_params": "{}", "features": ["NFCI_ret_5d__div__NVDA_vol_20d", "QCOM_ret_20d__zrel__PCE_zscore_60d", "vix_vs_ma20__zrel__XLK_Tech_zscore_60d", "spx_abs_ret_max_5d__ret5x__PCE_zscore_60d", "VIX_Price_zscore_60d__div__PCE_zscore_60d", "DAX_Germany_zscore_60d", "LOW_Lowes_ret_5d"], "is_new": false}, {"model_id": "v2_h7_CALM_RandomForest_N8", "algo": "RandomForest", "regime": "CALM", "horizon": 7, "n_features": 8, "F1_dir": 0.6159, "F1_UP_FORT": 0.3265, "F1_DOWN_FORT": 0.4086, "train_start": "2001-02-19", "sampler": "SMOTE", "best_params": "{}", "features": ["NFCI_ret_5d__div__NVDA_vol_20d", "QCOM_ret_20d__zrel__PCE_zscore_60d", "vix_vs_ma20__zrel__XLK_Tech_zscore_60d", "spx_abs_ret_max_5d__ret5x__PCE_zscore_60d", "VIX_Price_zscore_60d__div__PCE_zscore_60d", "DAX_Germany_zscore_60d", "LOW_Lowes_ret_5d", "CPB_CampbellSoup_vol_20d"], "is_new": false}, {"model_id": "v2_h7_CALM_RandomForest_N9", "algo": "RandomForest", "regime": "CALM", "horizon": 7, "n_features": 9, "F1_dir": 0.5793, "F1_UP_FORT": 0.3333, "F1_DOWN_FORT": 0.4211, "train_start": "2001-02-19", "sampler": "SMOTE", "best_params": "{}", "features": ["NFCI_ret_5d__div__NVDA_vol_20d", "QCOM_ret_20d__zrel__PCE_zscore_60d", "vix_vs_ma20__zrel__XLK_Tech_zscore_60d", "spx_abs_ret_max_5d__ret5x__PCE_zscore_60d", "VIX_Price_zscore_60d__div__PCE_zscore_60d", "DAX_Germany_zscore_60d", "LOW_Lowes_ret_5d", "CPB_CampbellSoup_vol_20d", "CI_Cigna_vol_20d"], "is_new": false}, {"model_id": "v2_h7_CALM_RandomForest_N10", "algo": "RandomForest", "regime": "CALM", "horizon": 7, "n_features": 10, "F1_dir": 0.5955, "F1_UP_FORT": 0.3448, "F1_DOWN_FORT": 0.4118, "train_start": "2001-02-19", "sampler": "SMOTE", "best_params": "{}", "features": ["NFCI_ret_5d__div__NVDA_vol_20d", "QCOM_ret_20d__zrel__PCE_zscore_60d", "vix_vs_ma20__zrel__XLK_Tech_zscore_60d", "spx_abs_ret_max_5d__ret5x__PCE_zscore_60d", "VIX_Price_zscore_60d__div__PCE_zscore_60d", "DAX_Germany_zscore_60d", "LOW_Lowes_ret_5d", "CPB_CampbellSoup_vol_20d", "CI_Cigna_vol_20d", "vix_vs_ma20__prod__NVDA_vol_20d"], "is_new": false}, {"model_id": "v2_h7_CALM_RandomForest_N11", "algo": "RandomForest", "regime": "CALM", "horizon": 7, "n_features": 11, "F1_dir": 0.6065, "F1_UP_FORT": 0.3505, "F1_DOWN_FORT": 0.4286, "train_start": "2001-02-19", "sampler": "SMOTE", "best_params": "{}", "features": ["NFCI_ret_5d__div__NVDA_vol_20d", "QCOM_ret_20d__zrel__PCE_zscore_60d", "vix_vs_ma20__zrel__XLK_Tech_zscore_60d", "spx_abs_ret_max_5d__ret5x__PCE_zscore_60d", "VIX_Price_zscore_60d__div__PCE_zscore_60d", "DAX_Germany_zscore_60d", "LOW_Lowes_ret_5d", "CPB_CampbellSoup_vol_20d", "CI_Cigna_vol_20d", "vix_vs_ma20__prod__NVDA_vol_20d", "VOD_Vodafone_zscore_60d"], "is_new": false}, {"model_id": "v2_h7_CALM_RandomForest_N12", "algo": "RandomForest", "regime": "CALM", "horizon": 7, "n_features": 12, "F1_dir": 0.6287, "F1_UP_FORT": 0.34, "F1_DOWN_FORT": 0.4242, "train_start": "2001-02-19", "sampler": "SMOTE", "best_params": "{}", "features": ["NFCI_ret_5d__div__NVDA_vol_20d", "QCOM_ret_20d__zrel__PCE_zscore_60d", "vix_vs_ma20__zrel__XLK_Tech_zscore_60d", "spx_abs_ret_max_5d__ret5x__PCE_zscore_60d", "VIX_Price_zscore_60d__div__PCE_zscore_60d", "DAX_Germany_zscore_60d", "LOW_Lowes_ret_5d", "CPB_CampbellSoup_vol_20d", "CI_Cigna_vol_20d", "vix_vs_ma20__prod__NVDA_vol_20d", "VOD_Vodafone_zscore_60d", "BA_ret_1d"], "is_new": false}, {"model_id": "v2_h7_CALM_LogisticRegression_N5", "algo": "LogisticRegression", "regime": "CALM", "horizon": 7, "n_features": 5, "F1_dir": 0.5193, "F1_UP_FORT": 0.2772, "F1_DOWN_FORT": 0.3761, "train_start": "2001-02-19", "sampler": "SMOTE", "best_params": "{}", "features": ["NFCI_ret_5d__div__NVDA_vol_20d", "QCOM_ret_20d__zrel__PCE_zscore_60d", "vix_vs_ma20__zrel__XLK_Tech_zscore_60d", "spx_abs_ret_max_5d__ret5x__PCE_zscore_60d", "VIX_Price_zscore_60d__div__PCE_zscore_60d"], "is_new": false}, {"model_id": "v2_h7_CALM_LogisticRegression_N6", "algo": "LogisticRegression", "regime": "CALM", "horizon": 7, "n_features": 6, "F1_dir": 0.5234, "F1_UP_FORT": 0.2745, "F1_DOWN_FORT": 0.3621, "train_start": "2001-02-19", "sampler": "SMOTE", "best_params": "{}", "features": ["NFCI_ret_5d__div__NVDA_vol_20d", "QCOM_ret_20d__zrel__PCE_zscore_60d", "vix_vs_ma20__zrel__XLK_Tech_zscore_60d", "spx_abs_ret_max_5d__ret5x__PCE_zscore_60d", "VIX_Price_zscore_60d__div__PCE_zscore_60d", "DAX_Germany_zscore_60d"], "is_new": false}, {"model_id": "v2_h7_CALM_LogisticRegression_N7", "algo": "LogisticRegression", "regime": "CALM", "horizon": 7, "n_features": 7, "F1_dir": 0.5494, "F1_UP_FORT": 0.3093, "F1_DOWN_FORT": 0.3902, "train_start": "2001-02-19", "sampler": "SMOTE", "best_params": "{}", "features": ["NFCI_ret_5d__div__NVDA_vol_20d", "QCOM_ret_20d__zrel__PCE_zscore_60d", "vix_vs_ma20__zrel__XLK_Tech_zscore_60d", "spx_abs_ret_max_5d__ret5x__PCE_zscore_60d", "VIX_Price_zscore_60d__div__PCE_zscore_60d", "DAX_Germany_zscore_60d", "LOW_Lowes_ret_5d"], "is_new": false}, {"model_id": "v2_h7_CALM_LogisticRegression_N8", "algo": "LogisticRegression", "regime": "CALM", "horizon": 7, "n_features": 8, "F1_dir": 0.5757, "F1_UP_FORT": 0.3019, "F1_DOWN_FORT": 0.3597, "train_start": "2001-02-19", "sampler": "SMOTE", "best_params": "{}", "features": ["NFCI_ret_5d__div__NVDA_vol_20d", "QCOM_ret_20d__zrel__PCE_zscore_60d", "vix_vs_ma20__zrel__XLK_Tech_zscore_60d", "spx_abs_ret_max_5d__ret5x__PCE_zscore_60d", "VIX_Price_zscore_60d__div__PCE_zscore_60d", "DAX_Germany_zscore_60d", "LOW_Lowes_ret_5d", "CPB_CampbellSoup_vol_20d"], "is_new": false}, {"model_id": "v2_h7_CALM_LogisticRegression_N9", "algo": "LogisticRegression", "regime": "CALM", "horizon": 7, "n_features": 9, "F1_dir": 0.6015, "F1_UP_FORT": 0.2957, "F1_DOWN_FORT": 0.3824, "train_start": "2001-02-19", "sampler": "SMOTE", "best_params": "{}", "features": ["NFCI_ret_5d__div__NVDA_vol_20d", "QCOM_ret_20d__zrel__PCE_zscore_60d", "vix_vs_ma20__zrel__XLK_Tech_zscore_60d", "spx_abs_ret_max_5d__ret5x__PCE_zscore_60d", "VIX_Price_zscore_60d__div__PCE_zscore_60d", "DAX_Germany_zscore_60d", "LOW_Lowes_ret_5d", "CPB_CampbellSoup_vol_20d", "CI_Cigna_vol_20d"], "is_new": false}, {"model_id": "v2_h7_CALM_LogisticRegression_N10", "algo": "LogisticRegression", "regime": "CALM", "horizon": 7, "n_features": 10, "F1_dir": 0.574, "F1_UP_FORT": 0.3704, "F1_DOWN_FORT": 0.3385, "train_start": "2001-02-19", "sampler": "SMOTE", "best_params": "{}", "features": ["NFCI_ret_5d__div__NVDA_vol_20d", "QCOM_ret_20d__zrel__PCE_zscore_60d", "vix_vs_ma20__zrel__XLK_Tech_zscore_60d", "spx_abs_ret_max_5d__ret5x__PCE_zscore_60d", "VIX_Price_zscore_60d__div__PCE_zscore_60d", "DAX_Germany_zscore_60d", "LOW_Lowes_ret_5d", "CPB_CampbellSoup_vol_20d", "CI_Cigna_vol_20d", "vix_vs_ma20__prod__NVDA_vol_20d"], "is_new": false}, {"model_id": "v2_h7_CALM_LogisticRegression_N11", "algo": "LogisticRegression", "regime": "CALM", "horizon": 7, "n_features": 11, "F1_dir": 0.5554, "F1_UP_FORT": 0.3208, "F1_DOWN_FORT": 0.3134, "train_start": "2001-02-19", "sampler": "SMOTE", "best_params": "{}", "features": ["NFCI_ret_5d__div__NVDA_vol_20d", "QCOM_ret_20d__zrel__PCE_zscore_60d", "vix_vs_ma20__zrel__XLK_Tech_zscore_60d", "spx_abs_ret_max_5d__ret5x__PCE_zscore_60d", "VIX_Price_zscore_60d__div__PCE_zscore_60d", "DAX_Germany_zscore_60d", "LOW_Lowes_ret_5d", "CPB_CampbellSoup_vol_20d", "CI_Cigna_vol_20d", "vix_vs_ma20__prod__NVDA_vol_20d", "VOD_Vodafone_zscore_60d"], "is_new": false}, {"model_id": "v2_h7_CALM_LogisticRegression_N12", "algo": "LogisticRegression", "regime": "CALM", "horizon": 7, "n_features": 12, "F1_dir": 0.5529, "F1_UP_FORT": 0.3238, "F1_DOWN_FORT": 0.3111, "train_start": "2001-02-19", "sampler": "SMOTE", "best_params": "{}", "features": ["NFCI_ret_5d__div__NVDA_vol_20d", "QCOM_ret_20d__zrel__PCE_zscore_60d", "vix_vs_ma20__zrel__XLK_Tech_zscore_60d", "spx_abs_ret_max_5d__ret5x__PCE_zscore_60d", "VIX_Price_zscore_60d__div__PCE_zscore_60d", "DAX_Germany_zscore_60d", "LOW_Lowes_ret_5d", "CPB_CampbellSoup_vol_20d", "CI_Cigna_vol_20d", "vix_vs_ma20__prod__NVDA_vol_20d", "VOD_Vodafone_zscore_60d", "BA_ret_1d"], "is_new": false}, {"model_id": "v2_h7_CALM_XGBoost_Optuna_N5", "algo": "XGBoost", "regime": "CALM", "horizon": 7, "n_features": 5, "F1_dir": 0.5497, "F1_UP_FORT": 0.325, "F1_DOWN_FORT": 0.4318, "train_start": "2001-02-19", "sampler": "SMOTE", "best_params": "{}", "features": ["NFCI_ret_5d__div__NVDA_vol_20d", "QCOM_ret_20d__zrel__PCE_zscore_60d", "vix_vs_ma20__zrel__XLK_Tech_zscore_60d", "spx_abs_ret_max_5d__ret5x__PCE_zscore_60d", "VIX_Price_zscore_60d__div__PCE_zscore_60d"], "is_new": false}, {"model_id": "v2_h7_CALM_XGBoost_OptunaCal_N5", "algo": "XGBoostCal", "regime": "CALM", "horizon": 7, "n_features": 5, "F1_dir": 0.5449, "F1_UP_FORT": 0.359, "F1_DOWN_FORT": 0.3158, "train_start": "2001-02-19", "sampler": "SMOTE", "best_params": "{}", "features": ["NFCI_ret_5d__div__NVDA_vol_20d", "QCOM_ret_20d__zrel__PCE_zscore_60d", "vix_vs_ma20__zrel__XLK_Tech_zscore_60d", "spx_abs_ret_max_5d__ret5x__PCE_zscore_60d", "VIX_Price_zscore_60d__div__PCE_zscore_60d"], "is_new": false}, {"model_id": "v2_h7_NORMAL_XGBoost_N5", "algo": "XGBoost", "regime": "NORMAL", "horizon": 7, "n_features": 5, "F1_dir": 0.5423, "F1_UP_FORT": 0.3409, "F1_DOWN_FORT": 0.415, "train_start": "2001-02-06", "sampler": "SMOTEENN", "best_params": "{}", "features": ["NFCI_ret_5d__minus__NFCI_vol_20d", "VIX_Price_zscore_60d__div__heston_var_ev_h1", "VIX_Price_zscore_60d__div__NFCI_vol_20d", "VIX_Price_zscore_60d__prod__Nikkei_Japan_vol_20d", "GD_GeneralDynamics_zscore_60d__ret5x__COST_zscore_60d"], "is_new": false}, {"model_id": "v2_h7_NORMAL_XGBoost_N6", "algo": "XGBoost", "regime": "NORMAL", "horizon": 7, "n_features": 6, "F1_dir": 0.5758, "F1_UP_FORT": 0.4316, "F1_DOWN_FORT": 0.4311, "train_start": "2001-02-06", "sampler": "SMOTEENN", "best_params": "{}", "features": ["NFCI_ret_5d__minus__NFCI_vol_20d", "VIX_Price_zscore_60d__div__heston_var_ev_h1", "VIX_Price_zscore_60d__div__NFCI_vol_20d", "VIX_Price_zscore_60d__prod__Nikkei_Japan_vol_20d", "GD_GeneralDynamics_zscore_60d__ret5x__COST_zscore_60d", "vix_vol_of_vol_10d__zrel__XLU_Util_ret_5d"], "is_new": false}, {"model_id": "v2_h7_NORMAL_XGBoost_N7", "algo": "XGBoost", "regime": "NORMAL", "horizon": 7, "n_features": 7, "F1_dir": 0.5635, "F1_UP_FORT": 0.3742, "F1_DOWN_FORT": 0.4204, "train_start": "2001-02-06", "sampler": "SMOTEENN", "best_params": "{}", "features": ["NFCI_ret_5d__minus__NFCI_vol_20d", "VIX_Price_zscore_60d__div__heston_var_ev_h1", "VIX_Price_zscore_60d__div__NFCI_vol_20d", "VIX_Price_zscore_60d__prod__Nikkei_Japan_vol_20d", "GD_GeneralDynamics_zscore_60d__ret5x__COST_zscore_60d", "vix_vol_of_vol_10d__zrel__XLU_Util_ret_5d", "HD_ret_20d"], "is_new": false}, {"model_id": "v2_h7_NORMAL_XGBoost_N8", "algo": "XGBoost", "regime": "NORMAL", "horizon": 7, "n_features": 8, "F1_dir": 0.5444, "F1_UP_FORT": 0.3926, "F1_DOWN_FORT": 0.4394, "train_start": "2001-02-06", "sampler": "SMOTEENN", "best_params": "{}", "features": ["NFCI_ret_5d__minus__NFCI_vol_20d", "VIX_Price_zscore_60d__div__heston_var_ev_h1", "VIX_Price_zscore_60d__div__NFCI_vol_20d", "VIX_Price_zscore_60d__prod__Nikkei_Japan_vol_20d", "GD_GeneralDynamics_zscore_60d__ret5x__COST_zscore_60d", "vix_vol_of_vol_10d__zrel__XLU_Util_ret_5d", "HD_ret_20d", "VIX_Price_zscore_60d__minus__GD_GeneralDynamics_zscore_60d"], "is_new": false}, {"model_id": "v2_h7_NORMAL_XGBoost_N9", "algo": "XGBoost", "regime": "NORMAL", "horizon": 7, "n_features": 9, "F1_dir": 0.5689, "F1_UP_FORT": 0.4174, "F1_DOWN_FORT": 0.444, "train_start": "2001-02-06", "sampler": "SMOTEENN", "best_params": "{}", "features": ["NFCI_ret_5d__minus__NFCI_vol_20d", "VIX_Price_zscore_60d__div__heston_var_ev_h1", "VIX_Price_zscore_60d__div__NFCI_vol_20d", "VIX_Price_zscore_60d__prod__Nikkei_Japan_vol_20d", "GD_GeneralDynamics_zscore_60d__ret5x__COST_zscore_60d", "vix_vol_of_vol_10d__zrel__XLU_Util_ret_5d", "HD_ret_20d", "VIX_Price_zscore_60d__minus__GD_GeneralDynamics_zscore_60d", "VIX_Price_zscore_60d__minus__CLX_Clorox_zscore_60d"], "is_new": false}, {"model_id": "v2_h7_NORMAL_XGBoost_N10", "algo": "XGBoost", "regime": "NORMAL", "horizon": 7, "n_features": 10, "F1_dir": 0.5672, "F1_UP_FORT": 0.4312, "F1_DOWN_FORT": 0.4359, "train_start": "2001-02-06", "sampler": "SMOTEENN", "best_params": "{}", "features": ["NFCI_ret_5d__minus__NFCI_vol_20d", "VIX_Price_zscore_60d__div__heston_var_ev_h1", "VIX_Price_zscore_60d__div__NFCI_vol_20d", "VIX_Price_zscore_60d__prod__Nikkei_Japan_vol_20d", "GD_GeneralDynamics_zscore_60d__ret5x__COST_zscore_60d", "vix_vol_of_vol_10d__zrel__XLU_Util_ret_5d", "HD_ret_20d", "VIX_Price_zscore_60d__minus__GD_GeneralDynamics_zscore_60d", "VIX_Price_zscore_60d__minus__CLX_Clorox_zscore_60d", "VIX_Price_zscore_60d__zrel__CLX_Clorox_zscore_60d"], "is_new": false}, {"model_id": "v2_h7_NORMAL_XGBoost_N11", "algo": "XGBoost", "regime": "NORMAL", "horizon": 7, "n_features": 11, "F1_dir": 0.5775, "F1_UP_FORT": 0.435, "F1_DOWN_FORT": 0.4449, "train_start": "2001-02-06", "sampler": "SMOTEENN", "best_params": "{}", "features": ["NFCI_ret_5d__minus__NFCI_vol_20d", "VIX_Price_zscore_60d__div__heston_var_ev_h1", "VIX_Price_zscore_60d__div__NFCI_vol_20d", "VIX_Price_zscore_60d__prod__Nikkei_Japan_vol_20d", "GD_GeneralDynamics_zscore_60d__ret5x__COST_zscore_60d", "vix_vol_of_vol_10d__zrel__XLU_Util_ret_5d", "HD_ret_20d", "VIX_Price_zscore_60d__minus__GD_GeneralDynamics_zscore_60d", "VIX_Price_zscore_60d__minus__CLX_Clorox_zscore_60d", "VIX_Price_zscore_60d__zrel__CLX_Clorox_zscore_60d", "vix_zscore_10d__macross__RTX_Raytheon_zscore_60d"], "is_new": false}, {"model_id": "v2_h7_NORMAL_XGBoost_N12", "algo": "XGBoost", "regime": "NORMAL", "horizon": 7, "n_features": 12, "F1_dir": 0.5684, "F1_UP_FORT": 0.4367, "F1_DOWN_FORT": 0.433, "train_start": "2001-02-06", "sampler": "SMOTEENN", "best_params": "{}", "features": ["NFCI_ret_5d__minus__NFCI_vol_20d", "VIX_Price_zscore_60d__div__heston_var_ev_h1", "VIX_Price_zscore_60d__div__NFCI_vol_20d", "VIX_Price_zscore_60d__prod__Nikkei_Japan_vol_20d", "GD_GeneralDynamics_zscore_60d__ret5x__COST_zscore_60d", "vix_vol_of_vol_10d__zrel__XLU_Util_ret_5d", "HD_ret_20d", "VIX_Price_zscore_60d__minus__GD_GeneralDynamics_zscore_60d", "VIX_Price_zscore_60d__minus__CLX_Clorox_zscore_60d", "VIX_Price_zscore_60d__zrel__CLX_Clorox_zscore_60d", "vix_zscore_10d__macross__RTX_Raytheon_zscore_60d", "vix_vol_of_vol_10d__minus__XLU_Util_ret_5d"], "is_new": false}, {"model_id": "v2_h7_NORMAL_XGBoost_N13", "algo": "XGBoost", "regime": "NORMAL", "horizon": 7, "n_features": 13, "F1_dir": 0.5834, "F1_UP_FORT": 0.4258, "F1_DOWN_FORT": 0.4148, "train_start": "2001-02-06", "sampler": "SMOTEENN", "best_params": "{}", "features": ["NFCI_ret_5d__minus__NFCI_vol_20d", "VIX_Price_zscore_60d__div__heston_var_ev_h1", "VIX_Price_zscore_60d__div__NFCI_vol_20d", "VIX_Price_zscore_60d__prod__Nikkei_Japan_vol_20d", "GD_GeneralDynamics_zscore_60d__ret5x__COST_zscore_60d", "vix_vol_of_vol_10d__zrel__XLU_Util_ret_5d", "HD_ret_20d", "VIX_Price_zscore_60d__minus__GD_GeneralDynamics_zscore_60d", "VIX_Price_zscore_60d__minus__CLX_Clorox_zscore_60d", "VIX_Price_zscore_60d__zrel__CLX_Clorox_zscore_60d", "vix_zscore_10d__macross__RTX_Raytheon_zscore_60d", "vix_vol_of_vol_10d__minus__XLU_Util_ret_5d", "Nikkei_Japan_vol_20d"], "is_new": false}, {"model_id": "v2_h7_NORMAL_XGBoost_N14", "algo": "XGBoost", "regime": "NORMAL", "horizon": 7, "n_features": 14, "F1_dir": 0.5988, "F1_UP_FORT": 0.4544, "F1_DOWN_FORT": 0.4298, "train_start": "2001-02-06", "sampler": "SMOTEENN", "best_params": "{}", "features": ["NFCI_ret_5d__minus__NFCI_vol_20d", "VIX_Price_zscore_60d__div__heston_var_ev_h1", "VIX_Price_zscore_60d__div__NFCI_vol_20d", "VIX_Price_zscore_60d__prod__Nikkei_Japan_vol_20d", "GD_GeneralDynamics_zscore_60d__ret5x__COST_zscore_60d", "vix_vol_of_vol_10d__zrel__XLU_Util_ret_5d", "HD_ret_20d", "VIX_Price_zscore_60d__minus__GD_GeneralDynamics_zscore_60d", "VIX_Price_zscore_60d__minus__CLX_Clorox_zscore_60d", "VIX_Price_zscore_60d__zrel__CLX_Clorox_zscore_60d", "vix_zscore_10d__macross__RTX_Raytheon_zscore_60d", "vix_vol_of_vol_10d__minus__XLU_Util_ret_5d", "Nikkei_Japan_vol_20d", "NFCI_ret_5d__div__NFCI_vol_20d"], "is_new": false}, {"model_id": "v2_h7_NORMAL_XGBoost_N15", "algo": "XGBoost", "regime": "NORMAL", "horizon": 7, "n_features": 15, "F1_dir": 0.5963, "F1_UP_FORT": 0.4686, "F1_DOWN_FORT": 0.4492, "train_start": "2001-02-06", "sampler": "SMOTEENN", "best_params": "{}", "features": ["NFCI_ret_5d__minus__NFCI_vol_20d", "VIX_Price_zscore_60d__div__heston_var_ev_h1", "VIX_Price_zscore_60d__div__NFCI_vol_20d", "VIX_Price_zscore_60d__prod__Nikkei_Japan_vol_20d", "GD_GeneralDynamics_zscore_60d__ret5x__COST_zscore_60d", "vix_vol_of_vol_10d__zrel__XLU_Util_ret_5d", "HD_ret_20d", "VIX_Price_zscore_60d__minus__GD_GeneralDynamics_zscore_60d", "VIX_Price_zscore_60d__minus__CLX_Clorox_zscore_60d", "VIX_Price_zscore_60d__zrel__CLX_Clorox_zscore_60d", "vix_zscore_10d__macross__RTX_Raytheon_zscore_60d", "vix_vol_of_vol_10d__minus__XLU_Util_ret_5d", "Nikkei_Japan_vol_20d", "NFCI_ret_5d__div__NFCI_vol_20d", "VIX_Price_zscore_60d__div__vix_vol_of_vol_10d"], "is_new": false}, {"model_id": "v2_h7_NORMAL_XGBoost_N16", "algo": "XGBoost", "regime": "NORMAL", "horizon": 7, "n_features": 16, "F1_dir": 0.5976, "F1_UP_FORT": 0.4586, "F1_DOWN_FORT": 0.4313, "train_start": "2001-02-06", "sampler": "SMOTEENN", "best_params": "{}", "features": ["NFCI_ret_5d__minus__NFCI_vol_20d", "VIX_Price_zscore_60d__div__heston_var_ev_h1", "VIX_Price_zscore_60d__div__NFCI_vol_20d", "VIX_Price_zscore_60d__prod__Nikkei_Japan_vol_20d", "GD_GeneralDynamics_zscore_60d__ret5x__COST_zscore_60d", "vix_vol_of_vol_10d__zrel__XLU_Util_ret_5d", "HD_ret_20d", "VIX_Price_zscore_60d__minus__GD_GeneralDynamics_zscore_60d", "VIX_Price_zscore_60d__minus__CLX_Clorox_zscore_60d", "VIX_Price_zscore_60d__zrel__CLX_Clorox_zscore_60d", "vix_zscore_10d__macross__RTX_Raytheon_zscore_60d", "vix_vol_of_vol_10d__minus__XLU_Util_ret_5d", "Nikkei_Japan_vol_20d", "NFCI_ret_5d__div__NFCI_vol_20d", "VIX_Price_zscore_60d__div__vix_vol_of_vol_10d", "VIX_Price_zscore_60d__zrel__GD_GeneralDynamics_zscore_60d"], "is_new": false}, {"model_id": "v2_h7_NORMAL_XGBoost_N17", "algo": "XGBoost", "regime": "NORMAL", "horizon": 7, "n_features": 17, "F1_dir": 0.6155, "F1_UP_FORT": 0.4706, "F1_DOWN_FORT": 0.4586, "train_start": "2001-02-06", "sampler": "SMOTEENN", "best_params": "{}", "features": ["NFCI_ret_5d__minus__NFCI_vol_20d", "VIX_Price_zscore_60d__div__heston_var_ev_h1", "VIX_Price_zscore_60d__div__NFCI_vol_20d", "VIX_Price_zscore_60d__prod__Nikkei_Japan_vol_20d", "GD_GeneralDynamics_zscore_60d__ret5x__COST_zscore_60d", "vix_vol_of_vol_10d__zrel__XLU_Util_ret_5d", "HD_ret_20d", "VIX_Price_zscore_60d__minus__GD_GeneralDynamics_zscore_60d", "VIX_Price_zscore_60d__minus__CLX_Clorox_zscore_60d", "VIX_Price_zscore_60d__zrel__CLX_Clorox_zscore_60d", "vix_zscore_10d__macross__RTX_Raytheon_zscore_60d", "vix_vol_of_vol_10d__minus__XLU_Util_ret_5d", "Nikkei_Japan_vol_20d", "NFCI_ret_5d__div__NFCI_vol_20d", "VIX_Price_zscore_60d__div__vix_vol_of_vol_10d", "VIX_Price_zscore_60d__zrel__GD_GeneralDynamics_zscore_60d", "vix_vol_of_vol_10d__prod__vix_level"], "is_new": false}, {"model_id": "v2_h7_NORMAL_XGBoost_N18", "algo": "XGBoost", "regime": "NORMAL", "horizon": 7, "n_features": 18, "F1_dir": 0.6168, "F1_UP_FORT": 0.4948, "F1_DOWN_FORT": 0.4568, "train_start": "2001-02-06", "sampler": "SMOTEENN", "best_params": "{}", "features": ["NFCI_ret_5d__minus__NFCI_vol_20d", "VIX_Price_zscore_60d__div__heston_var_ev_h1", "VIX_Price_zscore_60d__div__NFCI_vol_20d", "VIX_Price_zscore_60d__prod__Nikkei_Japan_vol_20d", "GD_GeneralDynamics_zscore_60d__ret5x__COST_zscore_60d", "vix_vol_of_vol_10d__zrel__XLU_Util_ret_5d", "HD_ret_20d", "VIX_Price_zscore_60d__minus__GD_GeneralDynamics_zscore_60d", "VIX_Price_zscore_60d__minus__CLX_Clorox_zscore_60d", "VIX_Price_zscore_60d__zrel__CLX_Clorox_zscore_60d", "vix_zscore_10d__macross__RTX_Raytheon_zscore_60d", "vix_vol_of_vol_10d__minus__XLU_Util_ret_5d", "Nikkei_Japan_vol_20d", "NFCI_ret_5d__div__NFCI_vol_20d", "VIX_Price_zscore_60d__div__vix_vol_of_vol_10d", "VIX_Price_zscore_60d__zrel__GD_GeneralDynamics_zscore_60d", "vix_vol_of_vol_10d__prod__vix_level", "vix_zscore_10d__zrel__EWP_Spain_zscore_60d"], "is_new": false}, {"model_id": "v2_h7_NORMAL_XGBoost_N19", "algo": "XGBoost", "regime": "NORMAL", "horizon": 7, "n_features": 19, "F1_dir": 0.606, "F1_UP_FORT": 0.4625, "F1_DOWN_FORT": 0.4636, "train_start": "2001-02-06", "sampler": "SMOTEENN", "best_params": "{}", "features": ["NFCI_ret_5d__minus__NFCI_vol_20d", "VIX_Price_zscore_60d__div__heston_var_ev_h1", "VIX_Price_zscore_60d__div__NFCI_vol_20d", "VIX_Price_zscore_60d__prod__Nikkei_Japan_vol_20d", "GD_GeneralDynamics_zscore_60d__ret5x__COST_zscore_60d", "vix_vol_of_vol_10d__zrel__XLU_Util_ret_5d", "HD_ret_20d", "VIX_Price_zscore_60d__minus__GD_GeneralDynamics_zscore_60d", "VIX_Price_zscore_60d__minus__CLX_Clorox_zscore_60d", "VIX_Price_zscore_60d__zrel__CLX_Clorox_zscore_60d", "vix_zscore_10d__macross__RTX_Raytheon_zscore_60d", "vix_vol_of_vol_10d__minus__XLU_Util_ret_5d", "Nikkei_Japan_vol_20d", "NFCI_ret_5d__div__NFCI_vol_20d", "VIX_Price_zscore_60d__div__vix_vol_of_vol_10d", "VIX_Price_zscore_60d__zrel__GD_GeneralDynamics_zscore_60d", "vix_vol_of_vol_10d__prod__vix_level", "vix_zscore_10d__zrel__EWP_Spain_zscore_60d", "IYR_US_REIT2_zscore_60d"], "is_new": false}, {"model_id": "v2_h7_NORMAL_XGBoost_N20", "algo": "XGBoost", "regime": "NORMAL", "horizon": 7, "n_features": 20, "F1_dir": 0.5883, "F1_UP_FORT": 0.457, "F1_DOWN_FORT": 0.4561, "train_start": "2001-02-06", "sampler": "SMOTEENN", "best_params": "{}", "features": ["NFCI_ret_5d__minus__NFCI_vol_20d", "VIX_Price_zscore_60d__div__heston_var_ev_h1", "VIX_Price_zscore_60d__div__NFCI_vol_20d", "VIX_Price_zscore_60d__prod__Nikkei_Japan_vol_20d", "GD_GeneralDynamics_zscore_60d__ret5x__COST_zscore_60d", "vix_vol_of_vol_10d__zrel__XLU_Util_ret_5d", "HD_ret_20d", "VIX_Price_zscore_60d__minus__GD_GeneralDynamics_zscore_60d", "VIX_Price_zscore_60d__minus__CLX_Clorox_zscore_60d", "VIX_Price_zscore_60d__zrel__CLX_Clorox_zscore_60d", "vix_zscore_10d__macross__RTX_Raytheon_zscore_60d", "vix_vol_of_vol_10d__minus__XLU_Util_ret_5d", "Nikkei_Japan_vol_20d", "NFCI_ret_5d__div__NFCI_vol_20d", "VIX_Price_zscore_60d__div__vix_vol_of_vol_10d", "VIX_Price_zscore_60d__zrel__GD_GeneralDynamics_zscore_60d", "vix_vol_of_vol_10d__prod__vix_level", "vix_zscore_10d__zrel__EWP_Spain_zscore_60d", "IYR_US_REIT2_zscore_60d", "heston_var_ev_h1__div__NFCI_vol_20d"], "is_new": false}, {"model_id": "v2_h7_NORMAL_LightGBM_N5", "algo": "LightGBM", "regime": "NORMAL", "horizon": 7, "n_features": 5, "F1_dir": 0.5359, "F1_UP_FORT": 0.3158, "F1_DOWN_FORT": 0.3991, "train_start": "2001-02-06", "sampler": "SMOTEENN", "best_params": "{}", "features": ["NFCI_ret_5d__minus__NFCI_vol_20d", "VIX_Price_zscore_60d__div__heston_var_ev_h1", "VIX_Price_zscore_60d__div__NFCI_vol_20d", "VIX_Price_zscore_60d__prod__Nikkei_Japan_vol_20d", "GD_GeneralDynamics_zscore_60d__ret5x__COST_zscore_60d"], "is_new": false}, {"model_id": "v2_h7_NORMAL_LightGBM_N6", "algo": "LightGBM", "regime": "NORMAL", "horizon": 7, "n_features": 6, "F1_dir": 0.5397, "F1_UP_FORT": 0.3434, "F1_DOWN_FORT": 0.4603, "train_start": "2001-02-06", "sampler": "SMOTEENN", "best_params": "{}", "features": ["NFCI_ret_5d__minus__NFCI_vol_20d", "VIX_Price_zscore_60d__div__heston_var_ev_h1", "VIX_Price_zscore_60d__div__NFCI_vol_20d", "VIX_Price_zscore_60d__prod__Nikkei_Japan_vol_20d", "GD_GeneralDynamics_zscore_60d__ret5x__COST_zscore_60d", "vix_vol_of_vol_10d__zrel__XLU_Util_ret_5d"], "is_new": false}, {"model_id": "v2_h7_NORMAL_LightGBM_N7", "algo": "LightGBM", "regime": "NORMAL", "horizon": 7, "n_features": 7, "F1_dir": 0.5529, "F1_UP_FORT": 0.3529, "F1_DOWN_FORT": 0.4439, "train_start": "2001-02-06", "sampler": "SMOTEENN", "best_params": "{}", "features": ["NFCI_ret_5d__minus__NFCI_vol_20d", "VIX_Price_zscore_60d__div__heston_var_ev_h1", "VIX_Price_zscore_60d__div__NFCI_vol_20d", "VIX_Price_zscore_60d__prod__Nikkei_Japan_vol_20d", "GD_GeneralDynamics_zscore_60d__ret5x__COST_zscore_60d", "vix_vol_of_vol_10d__zrel__XLU_Util_ret_5d", "HD_ret_20d"], "is_new": false}, {"model_id": "v2_h7_NORMAL_LightGBM_N8", "algo": "LightGBM", "regime": "NORMAL", "horizon": 7, "n_features": 8, "F1_dir": 0.5622, "F1_UP_FORT": 0.3373, "F1_DOWN_FORT": 0.4581, "train_start": "2001-02-06", "sampler": "SMOTEENN", "best_params": "{}", "features": ["NFCI_ret_5d__minus__NFCI_vol_20d", "VIX_Price_zscore_60d__div__heston_var_ev_h1", "VIX_Price_zscore_60d__div__NFCI_vol_20d", "VIX_Price_zscore_60d__prod__Nikkei_Japan_vol_20d", "GD_GeneralDynamics_zscore_60d__ret5x__COST_zscore_60d", "vix_vol_of_vol_10d__zrel__XLU_Util_ret_5d", "HD_ret_20d", "VIX_Price_zscore_60d__minus__GD_GeneralDynamics_zscore_60d"], "is_new": false}, {"model_id": "v2_h7_NORMAL_LightGBM_N9", "algo": "LightGBM", "regime": "NORMAL", "horizon": 7, "n_features": 9, "F1_dir": 0.5596, "F1_UP_FORT": 0.3649, "F1_DOWN_FORT": 0.462, "train_start": "2001-02-06", "sampler": "SMOTEENN", "best_params": "{}", "features": ["NFCI_ret_5d__minus__NFCI_vol_20d", "VIX_Price_zscore_60d__div__heston_var_ev_h1", "VIX_Price_zscore_60d__div__NFCI_vol_20d", "VIX_Price_zscore_60d__prod__Nikkei_Japan_vol_20d", "GD_GeneralDynamics_zscore_60d__ret5x__COST_zscore_60d", "vix_vol_of_vol_10d__zrel__XLU_Util_ret_5d", "HD_ret_20d", "VIX_Price_zscore_60d__minus__GD_GeneralDynamics_zscore_60d", "VIX_Price_zscore_60d__minus__CLX_Clorox_zscore_60d"], "is_new": false}, {"model_id": "v2_h7_NORMAL_LightGBM_N10", "algo": "LightGBM", "regime": "NORMAL", "horizon": 7, "n_features": 10, "F1_dir": 0.5596, "F1_UP_FORT": 0.3653, "F1_DOWN_FORT": 0.4235, "train_start": "2001-02-06", "sampler": "SMOTEENN", "best_params": "{}", "features": ["NFCI_ret_5d__minus__NFCI_vol_20d", "VIX_Price_zscore_60d__div__heston_var_ev_h1", "VIX_Price_zscore_60d__div__NFCI_vol_20d", "VIX_Price_zscore_60d__prod__Nikkei_Japan_vol_20d", "GD_GeneralDynamics_zscore_60d__ret5x__COST_zscore_60d", "vix_vol_of_vol_10d__zrel__XLU_Util_ret_5d", "HD_ret_20d", "VIX_Price_zscore_60d__minus__GD_GeneralDynamics_zscore_60d", "VIX_Price_zscore_60d__minus__CLX_Clorox_zscore_60d", "VIX_Price_zscore_60d__zrel__CLX_Clorox_zscore_60d"], "is_new": false}, {"model_id": "v2_h7_NORMAL_LightGBM_N11", "algo": "LightGBM", "regime": "NORMAL", "horizon": 7, "n_features": 11, "F1_dir": 0.5497, "F1_UP_FORT": 0.3675, "F1_DOWN_FORT": 0.4495, "train_start": "2001-02-06", "sampler": "SMOTEENN", "best_params": "{}", "features": ["NFCI_ret_5d__minus__NFCI_vol_20d", "VIX_Price_zscore_60d__div__heston_var_ev_h1", "VIX_Price_zscore_60d__div__NFCI_vol_20d", "VIX_Price_zscore_60d__prod__Nikkei_Japan_vol_20d", "GD_GeneralDynamics_zscore_60d__ret5x__COST_zscore_60d", "vix_vol_of_vol_10d__zrel__XLU_Util_ret_5d", "HD_ret_20d", "VIX_Price_zscore_60d__minus__GD_GeneralDynamics_zscore_60d", "VIX_Price_zscore_60d__minus__CLX_Clorox_zscore_60d", "VIX_Price_zscore_60d__zrel__CLX_Clorox_zscore_60d", "vix_zscore_10d__macross__RTX_Raytheon_zscore_60d"], "is_new": false}, {"model_id": "v2_h7_NORMAL_LightGBM_N12", "algo": "LightGBM", "regime": "NORMAL", "horizon": 7, "n_features": 12, "F1_dir": 0.563, "F1_UP_FORT": 0.3794, "F1_DOWN_FORT": 0.4498, "train_start": "2001-02-06", "sampler": "SMOTEENN", "best_params": "{}", "features": ["NFCI_ret_5d__minus__NFCI_vol_20d", "VIX_Price_zscore_60d__div__heston_var_ev_h1", "VIX_Price_zscore_60d__div__NFCI_vol_20d", "VIX_Price_zscore_60d__prod__Nikkei_Japan_vol_20d", "GD_GeneralDynamics_zscore_60d__ret5x__COST_zscore_60d", "vix_vol_of_vol_10d__zrel__XLU_Util_ret_5d", "HD_ret_20d", "VIX_Price_zscore_60d__minus__GD_GeneralDynamics_zscore_60d", "VIX_Price_zscore_60d__minus__CLX_Clorox_zscore_60d", "VIX_Price_zscore_60d__zrel__CLX_Clorox_zscore_60d", "vix_zscore_10d__macross__RTX_Raytheon_zscore_60d", "vix_vol_of_vol_10d__minus__XLU_Util_ret_5d"], "is_new": false}, {"model_id": "v2_h7_NORMAL_LightGBM_N13", "algo": "LightGBM", "regime": "NORMAL", "horizon": 7, "n_features": 13, "F1_dir": 0.5681, "F1_UP_FORT": 0.3458, "F1_DOWN_FORT": 0.4554, "train_start": "2001-02-06", "sampler": "SMOTEENN", "best_params": "{}", "features": ["NFCI_ret_5d__minus__NFCI_vol_20d", "VIX_Price_zscore_60d__div__heston_var_ev_h1", "VIX_Price_zscore_60d__div__NFCI_vol_20d", "VIX_Price_zscore_60d__prod__Nikkei_Japan_vol_20d", "GD_GeneralDynamics_zscore_60d__ret5x__COST_zscore_60d", "vix_vol_of_vol_10d__zrel__XLU_Util_ret_5d", "HD_ret_20d", "VIX_Price_zscore_60d__minus__GD_GeneralDynamics_zscore_60d", "VIX_Price_zscore_60d__minus__CLX_Clorox_zscore_60d", "VIX_Price_zscore_60d__zrel__CLX_Clorox_zscore_60d", "vix_zscore_10d__macross__RTX_Raytheon_zscore_60d", "vix_vol_of_vol_10d__minus__XLU_Util_ret_5d", "Nikkei_Japan_vol_20d"], "is_new": false}, {"model_id": "v2_h7_NORMAL_LightGBM_N14", "algo": "LightGBM", "regime": "NORMAL", "horizon": 7, "n_features": 14, "F1_dir": 0.5912, "F1_UP_FORT": 0.3917, "F1_DOWN_FORT": 0.4418, "train_start": "2001-02-06", "sampler": "SMOTEENN", "best_params": "{}", "features": ["NFCI_ret_5d__minus__NFCI_vol_20d", "VIX_Price_zscore_60d__div__heston_var_ev_h1", "VIX_Price_zscore_60d__div__NFCI_vol_20d", "VIX_Price_zscore_60d__prod__Nikkei_Japan_vol_20d", "GD_GeneralDynamics_zscore_60d__ret5x__COST_zscore_60d", "vix_vol_of_vol_10d__zrel__XLU_Util_ret_5d", "HD_ret_20d", "VIX_Price_zscore_60d__minus__GD_GeneralDynamics_zscore_60d", "VIX_Price_zscore_60d__minus__CLX_Clorox_zscore_60d", "VIX_Price_zscore_60d__zrel__CLX_Clorox_zscore_60d", "vix_zscore_10d__macross__RTX_Raytheon_zscore_60d", "vix_vol_of_vol_10d__minus__XLU_Util_ret_5d", "Nikkei_Japan_vol_20d", "NFCI_ret_5d__div__NFCI_vol_20d"], "is_new": false}, {"model_id": "v2_h7_NORMAL_LightGBM_N15", "algo": "LightGBM", "regime": "NORMAL", "horizon": 7, "n_features": 15, "F1_dir": 0.5821, "F1_UP_FORT": 0.4119, "F1_DOWN_FORT": 0.4581, "train_start": "2001-02-06", "sampler": "SMOTEENN", "best_params": "{}", "features": ["NFCI_ret_5d__minus__NFCI_vol_20d", "VIX_Price_zscore_60d__div__heston_var_ev_h1", "VIX_Price_zscore_60d__div__NFCI_vol_20d", "VIX_Price_zscore_60d__prod__Nikkei_Japan_vol_20d", "GD_GeneralDynamics_zscore_60d__ret5x__COST_zscore_60d", "vix_vol_of_vol_10d__zrel__XLU_Util_ret_5d", "HD_ret_20d", "VIX_Price_zscore_60d__minus__GD_GeneralDynamics_zscore_60d", "VIX_Price_zscore_60d__minus__CLX_Clorox_zscore_60d", "VIX_Price_zscore_60d__zrel__CLX_Clorox_zscore_60d", "vix_zscore_10d__macross__RTX_Raytheon_zscore_60d", "vix_vol_of_vol_10d__minus__XLU_Util_ret_5d", "Nikkei_Japan_vol_20d", "NFCI_ret_5d__div__NFCI_vol_20d", "VIX_Price_zscore_60d__div__vix_vol_of_vol_10d"], "is_new": false}, {"model_id": "v2_h7_NORMAL_LightGBM_N16", "algo": "LightGBM", "regime": "NORMAL", "horizon": 7, "n_features": 16, "F1_dir": 0.5749, "F1_UP_FORT": 0.4018, "F1_DOWN_FORT": 0.4412, "train_start": "2001-02-06", "sampler": "SMOTEENN", "best_params": "{}", "features": ["NFCI_ret_5d__minus__NFCI_vol_20d", "VIX_Price_zscore_60d__div__heston_var_ev_h1", "VIX_Price_zscore_60d__div__NFCI_vol_20d", "VIX_Price_zscore_60d__prod__Nikkei_Japan_vol_20d", "GD_GeneralDynamics_zscore_60d__ret5x__COST_zscore_60d", "vix_vol_of_vol_10d__zrel__XLU_Util_ret_5d", "HD_ret_20d", "VIX_Price_zscore_60d__minus__GD_GeneralDynamics_zscore_60d", "VIX_Price_zscore_60d__minus__CLX_Clorox_zscore_60d", "VIX_Price_zscore_60d__zrel__CLX_Clorox_zscore_60d", "vix_zscore_10d__macross__RTX_Raytheon_zscore_60d", "vix_vol_of_vol_10d__minus__XLU_Util_ret_5d", "Nikkei_Japan_vol_20d", "NFCI_ret_5d__div__NFCI_vol_20d", "VIX_Price_zscore_60d__div__vix_vol_of_vol_10d", "VIX_Price_zscore_60d__zrel__GD_GeneralDynamics_zscore_60d"], "is_new": false}, {"model_id": "v2_h7_NORMAL_LightGBM_N17", "algo": "LightGBM", "regime": "NORMAL", "horizon": 7, "n_features": 17, "F1_dir": 0.5746, "F1_UP_FORT": 0.414, "F1_DOWN_FORT": 0.4461, "train_start": "2001-02-06", "sampler": "SMOTEENN", "best_params": "{}", "features": ["NFCI_ret_5d__minus__NFCI_vol_20d", "VIX_Price_zscore_60d__div__heston_var_ev_h1", "VIX_Price_zscore_60d__div__NFCI_vol_20d", "VIX_Price_zscore_60d__prod__Nikkei_Japan_vol_20d", "GD_GeneralDynamics_zscore_60d__ret5x__COST_zscore_60d", "vix_vol_of_vol_10d__zrel__XLU_Util_ret_5d", "HD_ret_20d", "VIX_Price_zscore_60d__minus__GD_GeneralDynamics_zscore_60d", "VIX_Price_zscore_60d__minus__CLX_Clorox_zscore_60d", "VIX_Price_zscore_60d__zrel__CLX_Clorox_zscore_60d", "vix_zscore_10d__macross__RTX_Raytheon_zscore_60d", "vix_vol_of_vol_10d__minus__XLU_Util_ret_5d", "Nikkei_Japan_vol_20d", "NFCI_ret_5d__div__NFCI_vol_20d", "VIX_Price_zscore_60d__div__vix_vol_of_vol_10d", "VIX_Price_zscore_60d__zrel__GD_GeneralDynamics_zscore_60d", "vix_vol_of_vol_10d__prod__vix_level"], "is_new": false}, {"model_id": "v2_h7_NORMAL_LightGBM_N18", "algo": "LightGBM", "regime": "NORMAL", "horizon": 7, "n_features": 18, "F1_dir": 0.5941, "F1_UP_FORT": 0.4595, "F1_DOWN_FORT": 0.4513, "train_start": "2001-02-06", "sampler": "SMOTEENN", "best_params": "{}", "features": ["NFCI_ret_5d__minus__NFCI_vol_20d", "VIX_Price_zscore_60d__div__heston_var_ev_h1", "VIX_Price_zscore_60d__div__NFCI_vol_20d", "VIX_Price_zscore_60d__prod__Nikkei_Japan_vol_20d", "GD_GeneralDynamics_zscore_60d__ret5x__COST_zscore_60d", "vix_vol_of_vol_10d__zrel__XLU_Util_ret_5d", "HD_ret_20d", "VIX_Price_zscore_60d__minus__GD_GeneralDynamics_zscore_60d", "VIX_Price_zscore_60d__minus__CLX_Clorox_zscore_60d", "VIX_Price_zscore_60d__zrel__CLX_Clorox_zscore_60d", "vix_zscore_10d__macross__RTX_Raytheon_zscore_60d", "vix_vol_of_vol_10d__minus__XLU_Util_ret_5d", "Nikkei_Japan_vol_20d", "NFCI_ret_5d__div__NFCI_vol_20d", "VIX_Price_zscore_60d__div__vix_vol_of_vol_10d", "VIX_Price_zscore_60d__zrel__GD_GeneralDynamics_zscore_60d", "vix_vol_of_vol_10d__prod__vix_level", "vix_zscore_10d__zrel__EWP_Spain_zscore_60d"], "is_new": false}, {"model_id": "v2_h7_NORMAL_LightGBM_N19", "algo": "LightGBM", "regime": "NORMAL", "horizon": 7, "n_features": 19, "F1_dir": 0.5962, "F1_UP_FORT": 0.449, "F1_DOWN_FORT": 0.4423, "train_start": "2001-02-06", "sampler": "SMOTEENN", "best_params": "{}", "features": ["NFCI_ret_5d__minus__NFCI_vol_20d", "VIX_Price_zscore_60d__div__heston_var_ev_h1", "VIX_Price_zscore_60d__div__NFCI_vol_20d", "VIX_Price_zscore_60d__prod__Nikkei_Japan_vol_20d", "GD_GeneralDynamics_zscore_60d__ret5x__COST_zscore_60d", "vix_vol_of_vol_10d__zrel__XLU_Util_ret_5d", "HD_ret_20d", "VIX_Price_zscore_60d__minus__GD_GeneralDynamics_zscore_60d", "VIX_Price_zscore_60d__minus__CLX_Clorox_zscore_60d", "VIX_Price_zscore_60d__zrel__CLX_Clorox_zscore_60d", "vix_zscore_10d__macross__RTX_Raytheon_zscore_60d", "vix_vol_of_vol_10d__minus__XLU_Util_ret_5d", "Nikkei_Japan_vol_20d", "NFCI_ret_5d__div__NFCI_vol_20d", "VIX_Price_zscore_60d__div__vix_vol_of_vol_10d", "VIX_Price_zscore_60d__zrel__GD_GeneralDynamics_zscore_60d", "vix_vol_of_vol_10d__prod__vix_level", "vix_zscore_10d__zrel__EWP_Spain_zscore_60d", "IYR_US_REIT2_zscore_60d"], "is_new": false}, {"model_id": "v2_h7_NORMAL_LightGBM_N20", "algo": "LightGBM", "regime": "NORMAL", "horizon": 7, "n_features": 20, "F1_dir": 0.5913, "F1_UP_FORT": 0.4455, "F1_DOWN_FORT": 0.4899, "train_start": "2001-02-06", "sampler": "SMOTEENN", "best_params": "{}", "features": ["NFCI_ret_5d__minus__NFCI_vol_20d", "VIX_Price_zscore_60d__div__heston_var_ev_h1", "VIX_Price_zscore_60d__div__NFCI_vol_20d", "VIX_Price_zscore_60d__prod__Nikkei_Japan_vol_20d", "GD_GeneralDynamics_zscore_60d__ret5x__COST_zscore_60d", "vix_vol_of_vol_10d__zrel__XLU_Util_ret_5d", "HD_ret_20d", "VIX_Price_zscore_60d__minus__GD_GeneralDynamics_zscore_60d", "VIX_Price_zscore_60d__minus__CLX_Clorox_zscore_60d", "VIX_Price_zscore_60d__zrel__CLX_Clorox_zscore_60d", "vix_zscore_10d__macross__RTX_Raytheon_zscore_60d", "vix_vol_of_vol_10d__minus__XLU_Util_ret_5d", "Nikkei_Japan_vol_20d", "NFCI_ret_5d__div__NFCI_vol_20d", "VIX_Price_zscore_60d__div__vix_vol_of_vol_10d", "VIX_Price_zscore_60d__zrel__GD_GeneralDynamics_zscore_60d", "vix_vol_of_vol_10d__prod__vix_level", "vix_zscore_10d__zrel__EWP_Spain_zscore_60d", "IYR_US_REIT2_zscore_60d", "heston_var_ev_h1__div__NFCI_vol_20d"], "is_new": false}, {"model_id": "v2_h7_NORMAL_GradientBoosting_N5", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 7, "n_features": 5, "F1_dir": 0.5124, "F1_UP_FORT": 0.294, "F1_DOWN_FORT": 0.3905, "train_start": "2001-02-06", "sampler": "SMOTEENN", "best_params": "{}", "features": ["NFCI_ret_5d__minus__NFCI_vol_20d", "VIX_Price_zscore_60d__div__heston_var_ev_h1", "VIX_Price_zscore_60d__div__NFCI_vol_20d", "VIX_Price_zscore_60d__prod__Nikkei_Japan_vol_20d", "GD_GeneralDynamics_zscore_60d__ret5x__COST_zscore_60d"], "is_new": false}, {"model_id": "v2_h7_NORMAL_GradientBoosting_N6", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 7, "n_features": 6, "F1_dir": 0.5589, "F1_UP_FORT": 0.3786, "F1_DOWN_FORT": 0.4309, "train_start": "2001-02-06", "sampler": "SMOTEENN", "best_params": "{}", "features": ["NFCI_ret_5d__minus__NFCI_vol_20d", "VIX_Price_zscore_60d__div__heston_var_ev_h1", "VIX_Price_zscore_60d__div__NFCI_vol_20d", "VIX_Price_zscore_60d__prod__Nikkei_Japan_vol_20d", "GD_GeneralDynamics_zscore_60d__ret5x__COST_zscore_60d", "vix_vol_of_vol_10d__zrel__XLU_Util_ret_5d"], "is_new": false}, {"model_id": "v2_h7_NORMAL_GradientBoosting_N7", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 7, "n_features": 7, "F1_dir": 0.5689, "F1_UP_FORT": 0.3912, "F1_DOWN_FORT": 0.4306, "train_start": "2001-02-06", "sampler": "SMOTEENN", "best_params": "{}", "features": ["NFCI_ret_5d__minus__NFCI_vol_20d", "VIX_Price_zscore_60d__div__heston_var_ev_h1", "VIX_Price_zscore_60d__div__NFCI_vol_20d", "VIX_Price_zscore_60d__prod__Nikkei_Japan_vol_20d", "GD_GeneralDynamics_zscore_60d__ret5x__COST_zscore_60d", "vix_vol_of_vol_10d__zrel__XLU_Util_ret_5d", "HD_ret_20d"], "is_new": false}, {"model_id": "v2_h7_NORMAL_GradientBoosting_N8", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 7, "n_features": 8, "F1_dir": 0.5656, "F1_UP_FORT": 0.3956, "F1_DOWN_FORT": 0.4109, "train_start": "2001-02-06", "sampler": "SMOTEENN", "best_params": "{}", "features": ["NFCI_ret_5d__minus__NFCI_vol_20d", "VIX_Price_zscore_60d__div__heston_var_ev_h1", "VIX_Price_zscore_60d__div__NFCI_vol_20d", "VIX_Price_zscore_60d__prod__Nikkei_Japan_vol_20d", "GD_GeneralDynamics_zscore_60d__ret5x__COST_zscore_60d", "vix_vol_of_vol_10d__zrel__XLU_Util_ret_5d", "HD_ret_20d", "VIX_Price_zscore_60d__minus__GD_GeneralDynamics_zscore_60d"], "is_new": false}, {"model_id": "v2_h7_NORMAL_GradientBoosting_N9", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 7, "n_features": 9, "F1_dir": 0.5451, "F1_UP_FORT": 0.4034, "F1_DOWN_FORT": 0.4268, "train_start": "2001-02-06", "sampler": "SMOTEENN", "best_params": "{}", "features": ["NFCI_ret_5d__minus__NFCI_vol_20d", "VIX_Price_zscore_60d__div__heston_var_ev_h1", "VIX_Price_zscore_60d__div__NFCI_vol_20d", "VIX_Price_zscore_60d__prod__Nikkei_Japan_vol_20d", "GD_GeneralDynamics_zscore_60d__ret5x__COST_zscore_60d", "vix_vol_of_vol_10d__zrel__XLU_Util_ret_5d", "HD_ret_20d", "VIX_Price_zscore_60d__minus__GD_GeneralDynamics_zscore_60d", "VIX_Price_zscore_60d__minus__CLX_Clorox_zscore_60d"], "is_new": false}, {"model_id": "v2_h7_NORMAL_GradientBoosting_N10", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 7, "n_features": 10, "F1_dir": 0.5591, "F1_UP_FORT": 0.4228, "F1_DOWN_FORT": 0.4248, "train_start": "2001-02-06", "sampler": "SMOTEENN", "best_params": "{}", "features": ["NFCI_ret_5d__minus__NFCI_vol_20d", "VIX_Price_zscore_60d__div__heston_var_ev_h1", "VIX_Price_zscore_60d__div__NFCI_vol_20d", "VIX_Price_zscore_60d__prod__Nikkei_Japan_vol_20d", "GD_GeneralDynamics_zscore_60d__ret5x__COST_zscore_60d", "vix_vol_of_vol_10d__zrel__XLU_Util_ret_5d", "HD_ret_20d", "VIX_Price_zscore_60d__minus__GD_GeneralDynamics_zscore_60d", "VIX_Price_zscore_60d__minus__CLX_Clorox_zscore_60d", "VIX_Price_zscore_60d__zrel__CLX_Clorox_zscore_60d"], "is_new": false}, {"model_id": "v2_h7_NORMAL_GradientBoosting_N11", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 7, "n_features": 11, "F1_dir": 0.576, "F1_UP_FORT": 0.4042, "F1_DOWN_FORT": 0.4397, "train_start": "2001-02-06", "sampler": "SMOTEENN", "best_params": "{}", "features": ["NFCI_ret_5d__minus__NFCI_vol_20d", "VIX_Price_zscore_60d__div__heston_var_ev_h1", "VIX_Price_zscore_60d__div__NFCI_vol_20d", "VIX_Price_zscore_60d__prod__Nikkei_Japan_vol_20d", "GD_GeneralDynamics_zscore_60d__ret5x__COST_zscore_60d", "vix_vol_of_vol_10d__zrel__XLU_Util_ret_5d", "HD_ret_20d", "VIX_Price_zscore_60d__minus__GD_GeneralDynamics_zscore_60d", "VIX_Price_zscore_60d__minus__CLX_Clorox_zscore_60d", "VIX_Price_zscore_60d__zrel__CLX_Clorox_zscore_60d", "vix_zscore_10d__macross__RTX_Raytheon_zscore_60d"], "is_new": false}, {"model_id": "v2_h7_NORMAL_GradientBoosting_N12", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 7, "n_features": 12, "F1_dir": 0.5756, "F1_UP_FORT": 0.4268, "F1_DOWN_FORT": 0.4444, "train_start": "2001-02-06", "sampler": "SMOTEENN", "best_params": "{}", "features": ["NFCI_ret_5d__minus__NFCI_vol_20d", "VIX_Price_zscore_60d__div__heston_var_ev_h1", "VIX_Price_zscore_60d__div__NFCI_vol_20d", "VIX_Price_zscore_60d__prod__Nikkei_Japan_vol_20d", "GD_GeneralDynamics_zscore_60d__ret5x__COST_zscore_60d", "vix_vol_of_vol_10d__zrel__XLU_Util_ret_5d", "HD_ret_20d", "VIX_Price_zscore_60d__minus__GD_GeneralDynamics_zscore_60d", "VIX_Price_zscore_60d__minus__CLX_Clorox_zscore_60d", "VIX_Price_zscore_60d__zrel__CLX_Clorox_zscore_60d", "vix_zscore_10d__macross__RTX_Raytheon_zscore_60d", "vix_vol_of_vol_10d__minus__XLU_Util_ret_5d"], "is_new": false}, {"model_id": "v2_h7_NORMAL_GradientBoosting_N13", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 7, "n_features": 13, "F1_dir": 0.5751, "F1_UP_FORT": 0.4163, "F1_DOWN_FORT": 0.4548, "train_start": "2001-02-06", "sampler": "SMOTEENN", "best_params": "{}", "features": ["NFCI_ret_5d__minus__NFCI_vol_20d", "VIX_Price_zscore_60d__div__heston_var_ev_h1", "VIX_Price_zscore_60d__div__NFCI_vol_20d", "VIX_Price_zscore_60d__prod__Nikkei_Japan_vol_20d", "GD_GeneralDynamics_zscore_60d__ret5x__COST_zscore_60d", "vix_vol_of_vol_10d__zrel__XLU_Util_ret_5d", "HD_ret_20d", "VIX_Price_zscore_60d__minus__GD_GeneralDynamics_zscore_60d", "VIX_Price_zscore_60d__minus__CLX_Clorox_zscore_60d", "VIX_Price_zscore_60d__zrel__CLX_Clorox_zscore_60d", "vix_zscore_10d__macross__RTX_Raytheon_zscore_60d", "vix_vol_of_vol_10d__minus__XLU_Util_ret_5d", "Nikkei_Japan_vol_20d"], "is_new": false}, {"model_id": "v2_h7_NORMAL_GradientBoosting_N14", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 7, "n_features": 14, "F1_dir": 0.5869, "F1_UP_FORT": 0.4426, "F1_DOWN_FORT": 0.4439, "train_start": "2001-02-06", "sampler": "SMOTEENN", "best_params": "{}", "features": ["NFCI_ret_5d__minus__NFCI_vol_20d", "VIX_Price_zscore_60d__div__heston_var_ev_h1", "VIX_Price_zscore_60d__div__NFCI_vol_20d", "VIX_Price_zscore_60d__prod__Nikkei_Japan_vol_20d", "GD_GeneralDynamics_zscore_60d__ret5x__COST_zscore_60d", "vix_vol_of_vol_10d__zrel__XLU_Util_ret_5d", "HD_ret_20d", "VIX_Price_zscore_60d__minus__GD_GeneralDynamics_zscore_60d", "VIX_Price_zscore_60d__minus__CLX_Clorox_zscore_60d", "VIX_Price_zscore_60d__zrel__CLX_Clorox_zscore_60d", "vix_zscore_10d__macross__RTX_Raytheon_zscore_60d", "vix_vol_of_vol_10d__minus__XLU_Util_ret_5d", "Nikkei_Japan_vol_20d", "NFCI_ret_5d__div__NFCI_vol_20d"], "is_new": false}, {"model_id": "v2_h7_NORMAL_GradientBoosting_N15", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 7, "n_features": 15, "F1_dir": 0.5756, "F1_UP_FORT": 0.4262, "F1_DOWN_FORT": 0.4439, "train_start": "2001-02-06", "sampler": "SMOTEENN", "best_params": "{}", "features": ["NFCI_ret_5d__minus__NFCI_vol_20d", "VIX_Price_zscore_60d__div__heston_var_ev_h1", "VIX_Price_zscore_60d__div__NFCI_vol_20d", "VIX_Price_zscore_60d__prod__Nikkei_Japan_vol_20d", "GD_GeneralDynamics_zscore_60d__ret5x__COST_zscore_60d", "vix_vol_of_vol_10d__zrel__XLU_Util_ret_5d", "HD_ret_20d", "VIX_Price_zscore_60d__minus__GD_GeneralDynamics_zscore_60d", "VIX_Price_zscore_60d__minus__CLX_Clorox_zscore_60d", "VIX_Price_zscore_60d__zrel__CLX_Clorox_zscore_60d", "vix_zscore_10d__macross__RTX_Raytheon_zscore_60d", "vix_vol_of_vol_10d__minus__XLU_Util_ret_5d", "Nikkei_Japan_vol_20d", "NFCI_ret_5d__div__NFCI_vol_20d", "VIX_Price_zscore_60d__div__vix_vol_of_vol_10d"], "is_new": false}, {"model_id": "v2_h7_NORMAL_GradientBoosting_N16", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 7, "n_features": 16, "F1_dir": 0.5925, "F1_UP_FORT": 0.4686, "F1_DOWN_FORT": 0.4496, "train_start": "2001-02-06", "sampler": "SMOTEENN", "best_params": "{}", "features": ["NFCI_ret_5d__minus__NFCI_vol_20d", "VIX_Price_zscore_60d__div__heston_var_ev_h1", "VIX_Price_zscore_60d__div__NFCI_vol_20d", "VIX_Price_zscore_60d__prod__Nikkei_Japan_vol_20d", "GD_GeneralDynamics_zscore_60d__ret5x__COST_zscore_60d", "vix_vol_of_vol_10d__zrel__XLU_Util_ret_5d", "HD_ret_20d", "VIX_Price_zscore_60d__minus__GD_GeneralDynamics_zscore_60d", "VIX_Price_zscore_60d__minus__CLX_Clorox_zscore_60d", "VIX_Price_zscore_60d__zrel__CLX_Clorox_zscore_60d", "vix_zscore_10d__macross__RTX_Raytheon_zscore_60d", "vix_vol_of_vol_10d__minus__XLU_Util_ret_5d", "Nikkei_Japan_vol_20d", "NFCI_ret_5d__div__NFCI_vol_20d", "VIX_Price_zscore_60d__div__vix_vol_of_vol_10d", "VIX_Price_zscore_60d__zrel__GD_GeneralDynamics_zscore_60d"], "is_new": false}, {"model_id": "v2_h7_NORMAL_GradientBoosting_N17", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 7, "n_features": 17, "F1_dir": 0.5832, "F1_UP_FORT": 0.4619, "F1_DOWN_FORT": 0.4513, "train_start": "2001-02-06", "sampler": "SMOTEENN", "best_params": "{}", "features": ["NFCI_ret_5d__minus__NFCI_vol_20d", "VIX_Price_zscore_60d__div__heston_var_ev_h1", "VIX_Price_zscore_60d__div__NFCI_vol_20d", "VIX_Price_zscore_60d__prod__Nikkei_Japan_vol_20d", "GD_GeneralDynamics_zscore_60d__ret5x__COST_zscore_60d", "vix_vol_of_vol_10d__zrel__XLU_Util_ret_5d", "HD_ret_20d", "VIX_Price_zscore_60d__minus__GD_GeneralDynamics_zscore_60d", "VIX_Price_zscore_60d__minus__CLX_Clorox_zscore_60d", "VIX_Price_zscore_60d__zrel__CLX_Clorox_zscore_60d", "vix_zscore_10d__macross__RTX_Raytheon_zscore_60d", "vix_vol_of_vol_10d__minus__XLU_Util_ret_5d", "Nikkei_Japan_vol_20d", "NFCI_ret_5d__div__NFCI_vol_20d", "VIX_Price_zscore_60d__div__vix_vol_of_vol_10d", "VIX_Price_zscore_60d__zrel__GD_GeneralDynamics_zscore_60d", "vix_vol_of_vol_10d__prod__vix_level"], "is_new": false}, {"model_id": "v2_h7_NORMAL_GradientBoosting_N18", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 7, "n_features": 18, "F1_dir": 0.6142, "F1_UP_FORT": 0.4899, "F1_DOWN_FORT": 0.4677, "train_start": "2001-02-06", "sampler": "SMOTEENN", "best_params": "{}", "features": ["NFCI_ret_5d__minus__NFCI_vol_20d", "VIX_Price_zscore_60d__div__heston_var_ev_h1", "VIX_Price_zscore_60d__div__NFCI_vol_20d", "VIX_Price_zscore_60d__prod__Nikkei_Japan_vol_20d", "GD_GeneralDynamics_zscore_60d__ret5x__COST_zscore_60d", "vix_vol_of_vol_10d__zrel__XLU_Util_ret_5d", "HD_ret_20d", "VIX_Price_zscore_60d__minus__GD_GeneralDynamics_zscore_60d", "VIX_Price_zscore_60d__minus__CLX_Clorox_zscore_60d", "VIX_Price_zscore_60d__zrel__CLX_Clorox_zscore_60d", "vix_zscore_10d__macross__RTX_Raytheon_zscore_60d", "vix_vol_of_vol_10d__minus__XLU_Util_ret_5d", "Nikkei_Japan_vol_20d", "NFCI_ret_5d__div__NFCI_vol_20d", "VIX_Price_zscore_60d__div__vix_vol_of_vol_10d", "VIX_Price_zscore_60d__zrel__GD_GeneralDynamics_zscore_60d", "vix_vol_of_vol_10d__prod__vix_level", "vix_zscore_10d__zrel__EWP_Spain_zscore_60d"], "is_new": false}, {"model_id": "v2_h7_NORMAL_GradientBoosting_N19", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 7, "n_features": 19, "F1_dir": 0.6071, "F1_UP_FORT": 0.4625, "F1_DOWN_FORT": 0.458, "train_start": "2001-02-06", "sampler": "SMOTEENN", "best_params": "{}", "features": ["NFCI_ret_5d__minus__NFCI_vol_20d", "VIX_Price_zscore_60d__div__heston_var_ev_h1", "VIX_Price_zscore_60d__div__NFCI_vol_20d", "VIX_Price_zscore_60d__prod__Nikkei_Japan_vol_20d", "GD_GeneralDynamics_zscore_60d__ret5x__COST_zscore_60d", "vix_vol_of_vol_10d__zrel__XLU_Util_ret_5d", "HD_ret_20d", "VIX_Price_zscore_60d__minus__GD_GeneralDynamics_zscore_60d", "VIX_Price_zscore_60d__minus__CLX_Clorox_zscore_60d", "VIX_Price_zscore_60d__zrel__CLX_Clorox_zscore_60d", "vix_zscore_10d__macross__RTX_Raytheon_zscore_60d", "vix_vol_of_vol_10d__minus__XLU_Util_ret_5d", "Nikkei_Japan_vol_20d", "NFCI_ret_5d__div__NFCI_vol_20d", "VIX_Price_zscore_60d__div__vix_vol_of_vol_10d", "VIX_Price_zscore_60d__zrel__GD_GeneralDynamics_zscore_60d", "vix_vol_of_vol_10d__prod__vix_level", "vix_zscore_10d__zrel__EWP_Spain_zscore_60d", "IYR_US_REIT2_zscore_60d"], "is_new": false}, {"model_id": "v2_h7_NORMAL_GradientBoosting_N20", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 7, "n_features": 20, "F1_dir": 0.5922, "F1_UP_FORT": 0.4671, "F1_DOWN_FORT": 0.4854, "train_start": "2001-02-06", "sampler": "SMOTEENN", "best_params": "{}", "features": ["NFCI_ret_5d__minus__NFCI_vol_20d", "VIX_Price_zscore_60d__div__heston_var_ev_h1", "VIX_Price_zscore_60d__div__NFCI_vol_20d", "VIX_Price_zscore_60d__prod__Nikkei_Japan_vol_20d", "GD_GeneralDynamics_zscore_60d__ret5x__COST_zscore_60d", "vix_vol_of_vol_10d__zrel__XLU_Util_ret_5d", "HD_ret_20d", "VIX_Price_zscore_60d__minus__GD_GeneralDynamics_zscore_60d", "VIX_Price_zscore_60d__minus__CLX_Clorox_zscore_60d", "VIX_Price_zscore_60d__zrel__CLX_Clorox_zscore_60d", "vix_zscore_10d__macross__RTX_Raytheon_zscore_60d", "vix_vol_of_vol_10d__minus__XLU_Util_ret_5d", "Nikkei_Japan_vol_20d", "NFCI_ret_5d__div__NFCI_vol_20d", "VIX_Price_zscore_60d__div__vix_vol_of_vol_10d", "VIX_Price_zscore_60d__zrel__GD_GeneralDynamics_zscore_60d", "vix_vol_of_vol_10d__prod__vix_level", "vix_zscore_10d__zrel__EWP_Spain_zscore_60d", "IYR_US_REIT2_zscore_60d", "heston_var_ev_h1__div__NFCI_vol_20d"], "is_new": false}, {"model_id": "v2_h7_NORMAL_RandomForest_N5", "algo": "RandomForest", "regime": "NORMAL", "horizon": 7, "n_features": 5, "F1_dir": 0.513, "F1_UP_FORT": 0.2908, "F1_DOWN_FORT": 0.419, "train_start": "2001-02-06", "sampler": "SMOTEENN", "best_params": "{}", "features": ["NFCI_ret_5d__minus__NFCI_vol_20d", "VIX_Price_zscore_60d__div__heston_var_ev_h1", "VIX_Price_zscore_60d__div__NFCI_vol_20d", "VIX_Price_zscore_60d__prod__Nikkei_Japan_vol_20d", "GD_GeneralDynamics_zscore_60d__ret5x__COST_zscore_60d"], "is_new": false}, {"model_id": "v2_h7_NORMAL_RandomForest_N6", "algo": "RandomForest", "regime": "NORMAL", "horizon": 7, "n_features": 6, "F1_dir": 0.5118, "F1_UP_FORT": 0.3116, "F1_DOWN_FORT": 0.4477, "train_start": "2001-02-06", "sampler": "SMOTEENN", "best_params": "{}", "features": ["NFCI_ret_5d__minus__NFCI_vol_20d", "VIX_Price_zscore_60d__div__heston_var_ev_h1", "VIX_Price_zscore_60d__div__NFCI_vol_20d", "VIX_Price_zscore_60d__prod__Nikkei_Japan_vol_20d", "GD_GeneralDynamics_zscore_60d__ret5x__COST_zscore_60d", "vix_vol_of_vol_10d__zrel__XLU_Util_ret_5d"], "is_new": false}, {"model_id": "v2_h7_NORMAL_RandomForest_N7", "algo": "RandomForest", "regime": "NORMAL", "horizon": 7, "n_features": 7, "F1_dir": 0.5407, "F1_UP_FORT": 0.3641, "F1_DOWN_FORT": 0.4691, "train_start": "2001-02-06", "sampler": "SMOTEENN", "best_params": "{}", "features": ["NFCI_ret_5d__minus__NFCI_vol_20d", "VIX_Price_zscore_60d__div__heston_var_ev_h1", "VIX_Price_zscore_60d__div__NFCI_vol_20d", "VIX_Price_zscore_60d__prod__Nikkei_Japan_vol_20d", "GD_GeneralDynamics_zscore_60d__ret5x__COST_zscore_60d", "vix_vol_of_vol_10d__zrel__XLU_Util_ret_5d", "HD_ret_20d"], "is_new": false}, {"model_id": "v2_h7_NORMAL_RandomForest_N8", "algo": "RandomForest", "regime": "NORMAL", "horizon": 7, "n_features": 8, "F1_dir": 0.5416, "F1_UP_FORT": 0.3298, "F1_DOWN_FORT": 0.4468, "train_start": "2001-02-06", "sampler": "SMOTEENN", "best_params": "{}", "features": ["NFCI_ret_5d__minus__NFCI_vol_20d", "VIX_Price_zscore_60d__div__heston_var_ev_h1", "VIX_Price_zscore_60d__div__NFCI_vol_20d", "VIX_Price_zscore_60d__prod__Nikkei_Japan_vol_20d", "GD_GeneralDynamics_zscore_60d__ret5x__COST_zscore_60d", "vix_vol_of_vol_10d__zrel__XLU_Util_ret_5d", "HD_ret_20d", "VIX_Price_zscore_60d__minus__GD_GeneralDynamics_zscore_60d"], "is_new": false}, {"model_id": "v2_h7_NORMAL_RandomForest_N9", "algo": "RandomForest", "regime": "NORMAL", "horizon": 7, "n_features": 9, "F1_dir": 0.5395, "F1_UP_FORT": 0.3102, "F1_DOWN_FORT": 0.437, "train_start": "2001-02-06", "sampler": "SMOTEENN", "best_params": "{}", "features": ["NFCI_ret_5d__minus__NFCI_vol_20d", "VIX_Price_zscore_60d__div__heston_var_ev_h1", "VIX_Price_zscore_60d__div__NFCI_vol_20d", "VIX_Price_zscore_60d__prod__Nikkei_Japan_vol_20d", "GD_GeneralDynamics_zscore_60d__ret5x__COST_zscore_60d", "vix_vol_of_vol_10d__zrel__XLU_Util_ret_5d", "HD_ret_20d", "VIX_Price_zscore_60d__minus__GD_GeneralDynamics_zscore_60d", "VIX_Price_zscore_60d__minus__CLX_Clorox_zscore_60d"], "is_new": false}, {"model_id": "v2_h7_NORMAL_RandomForest_N10", "algo": "RandomForest", "regime": "NORMAL", "horizon": 7, "n_features": 10, "F1_dir": 0.5595, "F1_UP_FORT": 0.368, "F1_DOWN_FORT": 0.4557, "train_start": "2001-02-06", "sampler": "SMOTEENN", "best_params": "{}", "features": ["NFCI_ret_5d__minus__NFCI_vol_20d", "VIX_Price_zscore_60d__div__heston_var_ev_h1", "VIX_Price_zscore_60d__div__NFCI_vol_20d", "VIX_Price_zscore_60d__prod__Nikkei_Japan_vol_20d", "GD_GeneralDynamics_zscore_60d__ret5x__COST_zscore_60d", "vix_vol_of_vol_10d__zrel__XLU_Util_ret_5d", "HD_ret_20d", "VIX_Price_zscore_60d__minus__GD_GeneralDynamics_zscore_60d", "VIX_Price_zscore_60d__minus__CLX_Clorox_zscore_60d", "VIX_Price_zscore_60d__zrel__CLX_Clorox_zscore_60d"], "is_new": false}, {"model_id": "v2_h7_NORMAL_RandomForest_N11", "algo": "RandomForest", "regime": "NORMAL", "horizon": 7, "n_features": 11, "F1_dir": 0.5723, "F1_UP_FORT": 0.4105, "F1_DOWN_FORT": 0.4619, "train_start": "2001-02-06", "sampler": "SMOTEENN", "best_params": "{}", "features": ["NFCI_ret_5d__minus__NFCI_vol_20d", "VIX_Price_zscore_60d__div__heston_var_ev_h1", "VIX_Price_zscore_60d__div__NFCI_vol_20d", "VIX_Price_zscore_60d__prod__Nikkei_Japan_vol_20d", "GD_GeneralDynamics_zscore_60d__ret5x__COST_zscore_60d", "vix_vol_of_vol_10d__zrel__XLU_Util_ret_5d", "HD_ret_20d", "VIX_Price_zscore_60d__minus__GD_GeneralDynamics_zscore_60d", "VIX_Price_zscore_60d__minus__CLX_Clorox_zscore_60d", "VIX_Price_zscore_60d__zrel__CLX_Clorox_zscore_60d", "vix_zscore_10d__macross__RTX_Raytheon_zscore_60d"], "is_new": false}, {"model_id": "v2_h7_NORMAL_RandomForest_N12", "algo": "RandomForest", "regime": "NORMAL", "horizon": 7, "n_features": 12, "F1_dir": 0.555, "F1_UP_FORT": 0.397, "F1_DOWN_FORT": 0.4372, "train_start": "2001-02-06", "sampler": "SMOTEENN", "best_params": "{}", "features": ["NFCI_ret_5d__minus__NFCI_vol_20d", "VIX_Price_zscore_60d__div__heston_var_ev_h1", "VIX_Price_zscore_60d__div__NFCI_vol_20d", "VIX_Price_zscore_60d__prod__Nikkei_Japan_vol_20d", "GD_GeneralDynamics_zscore_60d__ret5x__COST_zscore_60d", "vix_vol_of_vol_10d__zrel__XLU_Util_ret_5d", "HD_ret_20d", "VIX_Price_zscore_60d__minus__GD_GeneralDynamics_zscore_60d", "VIX_Price_zscore_60d__minus__CLX_Clorox_zscore_60d", "VIX_Price_zscore_60d__zrel__CLX_Clorox_zscore_60d", "vix_zscore_10d__macross__RTX_Raytheon_zscore_60d", "vix_vol_of_vol_10d__minus__XLU_Util_ret_5d"], "is_new": false}, {"model_id": "v2_h7_NORMAL_RandomForest_N13", "algo": "RandomForest", "regime": "NORMAL", "horizon": 7, "n_features": 13, "F1_dir": 0.56, "F1_UP_FORT": 0.371, "F1_DOWN_FORT": 0.4502, "train_start": "2001-02-06", "sampler": "SMOTEENN", "best_params": "{}", "features": ["NFCI_ret_5d__minus__NFCI_vol_20d", "VIX_Price_zscore_60d__div__heston_var_ev_h1", "VIX_Price_zscore_60d__div__NFCI_vol_20d", "VIX_Price_zscore_60d__prod__Nikkei_Japan_vol_20d", "GD_GeneralDynamics_zscore_60d__ret5x__COST_zscore_60d", "vix_vol_of_vol_10d__zrel__XLU_Util_ret_5d", "HD_ret_20d", "VIX_Price_zscore_60d__minus__GD_GeneralDynamics_zscore_60d", "VIX_Price_zscore_60d__minus__CLX_Clorox_zscore_60d", "VIX_Price_zscore_60d__zrel__CLX_Clorox_zscore_60d", "vix_zscore_10d__macross__RTX_Raytheon_zscore_60d", "vix_vol_of_vol_10d__minus__XLU_Util_ret_5d", "Nikkei_Japan_vol_20d"], "is_new": false}, {"model_id": "v2_h7_NORMAL_RandomForest_N14", "algo": "RandomForest", "regime": "NORMAL", "horizon": 7, "n_features": 14, "F1_dir": 0.5808, "F1_UP_FORT": 0.3611, "F1_DOWN_FORT": 0.4578, "train_start": "2001-02-06", "sampler": "SMOTEENN", "best_params": "{}", "features": ["NFCI_ret_5d__minus__NFCI_vol_20d", "VIX_Price_zscore_60d__div__heston_var_ev_h1", "VIX_Price_zscore_60d__div__NFCI_vol_20d", "VIX_Price_zscore_60d__prod__Nikkei_Japan_vol_20d", "GD_GeneralDynamics_zscore_60d__ret5x__COST_zscore_60d", "vix_vol_of_vol_10d__zrel__XLU_Util_ret_5d", "HD_ret_20d", "VIX_Price_zscore_60d__minus__GD_GeneralDynamics_zscore_60d", "VIX_Price_zscore_60d__minus__CLX_Clorox_zscore_60d", "VIX_Price_zscore_60d__zrel__CLX_Clorox_zscore_60d", "vix_zscore_10d__macross__RTX_Raytheon_zscore_60d", "vix_vol_of_vol_10d__minus__XLU_Util_ret_5d", "Nikkei_Japan_vol_20d", "NFCI_ret_5d__div__NFCI_vol_20d"], "is_new": false}, {"model_id": "v2_h7_NORMAL_RandomForest_N15", "algo": "RandomForest", "regime": "NORMAL", "horizon": 7, "n_features": 15, "F1_dir": 0.5738, "F1_UP_FORT": 0.349, "F1_DOWN_FORT": 0.472, "train_start": "2001-02-06", "sampler": "SMOTEENN", "best_params": "{}", "features": ["NFCI_ret_5d__minus__NFCI_vol_20d", "VIX_Price_zscore_60d__div__heston_var_ev_h1", "VIX_Price_zscore_60d__div__NFCI_vol_20d", "VIX_Price_zscore_60d__prod__Nikkei_Japan_vol_20d", "GD_GeneralDynamics_zscore_60d__ret5x__COST_zscore_60d", "vix_vol_of_vol_10d__zrel__XLU_Util_ret_5d", "HD_ret_20d", "VIX_Price_zscore_60d__minus__GD_GeneralDynamics_zscore_60d", "VIX_Price_zscore_60d__minus__CLX_Clorox_zscore_60d", "VIX_Price_zscore_60d__zrel__CLX_Clorox_zscore_60d", "vix_zscore_10d__macross__RTX_Raytheon_zscore_60d", "vix_vol_of_vol_10d__minus__XLU_Util_ret_5d", "Nikkei_Japan_vol_20d", "NFCI_ret_5d__div__NFCI_vol_20d", "VIX_Price_zscore_60d__div__vix_vol_of_vol_10d"], "is_new": false}, {"model_id": "v2_h7_NORMAL_RandomForest_N16", "algo": "RandomForest", "regime": "NORMAL", "horizon": 7, "n_features": 16, "F1_dir": 0.576, "F1_UP_FORT": 0.3704, "F1_DOWN_FORT": 0.4709, "train_start": "2001-02-06", "sampler": "SMOTEENN", "best_params": "{}", "features": ["NFCI_ret_5d__minus__NFCI_vol_20d", "VIX_Price_zscore_60d__div__heston_var_ev_h1", "VIX_Price_zscore_60d__div__NFCI_vol_20d", "VIX_Price_zscore_60d__prod__Nikkei_Japan_vol_20d", "GD_GeneralDynamics_zscore_60d__ret5x__COST_zscore_60d", "vix_vol_of_vol_10d__zrel__XLU_Util_ret_5d", "HD_ret_20d", "VIX_Price_zscore_60d__minus__GD_GeneralDynamics_zscore_60d", "VIX_Price_zscore_60d__minus__CLX_Clorox_zscore_60d", "VIX_Price_zscore_60d__zrel__CLX_Clorox_zscore_60d", "vix_zscore_10d__macross__RTX_Raytheon_zscore_60d", "vix_vol_of_vol_10d__minus__XLU_Util_ret_5d", "Nikkei_Japan_vol_20d", "NFCI_ret_5d__div__NFCI_vol_20d", "VIX_Price_zscore_60d__div__vix_vol_of_vol_10d", "VIX_Price_zscore_60d__zrel__GD_GeneralDynamics_zscore_60d"], "is_new": false}, {"model_id": "v2_h7_NORMAL_RandomForest_N17", "algo": "RandomForest", "regime": "NORMAL", "horizon": 7, "n_features": 17, "F1_dir": 0.5764, "F1_UP_FORT": 0.3784, "F1_DOWN_FORT": 0.4524, "train_start": "2001-02-06", "sampler": "SMOTEENN", "best_params": "{}", "features": ["NFCI_ret_5d__minus__NFCI_vol_20d", "VIX_Price_zscore_60d__div__heston_var_ev_h1", "VIX_Price_zscore_60d__div__NFCI_vol_20d", "VIX_Price_zscore_60d__prod__Nikkei_Japan_vol_20d", "GD_GeneralDynamics_zscore_60d__ret5x__COST_zscore_60d", "vix_vol_of_vol_10d__zrel__XLU_Util_ret_5d", "HD_ret_20d", "VIX_Price_zscore_60d__minus__GD_GeneralDynamics_zscore_60d", "VIX_Price_zscore_60d__minus__CLX_Clorox_zscore_60d", "VIX_Price_zscore_60d__zrel__CLX_Clorox_zscore_60d", "vix_zscore_10d__macross__RTX_Raytheon_zscore_60d", "vix_vol_of_vol_10d__minus__XLU_Util_ret_5d", "Nikkei_Japan_vol_20d", "NFCI_ret_5d__div__NFCI_vol_20d", "VIX_Price_zscore_60d__div__vix_vol_of_vol_10d", "VIX_Price_zscore_60d__zrel__GD_GeneralDynamics_zscore_60d", "vix_vol_of_vol_10d__prod__vix_level"], "is_new": false}, {"model_id": "v2_h7_NORMAL_RandomForest_N18", "algo": "RandomForest", "regime": "NORMAL", "horizon": 7, "n_features": 18, "F1_dir": 0.5995, "F1_UP_FORT": 0.4053, "F1_DOWN_FORT": 0.4838, "train_start": "2001-02-06", "sampler": "SMOTEENN", "best_params": "{}", "features": ["NFCI_ret_5d__minus__NFCI_vol_20d", "VIX_Price_zscore_60d__div__heston_var_ev_h1", "VIX_Price_zscore_60d__div__NFCI_vol_20d", "VIX_Price_zscore_60d__prod__Nikkei_Japan_vol_20d", "GD_GeneralDynamics_zscore_60d__ret5x__COST_zscore_60d", "vix_vol_of_vol_10d__zrel__XLU_Util_ret_5d", "HD_ret_20d", "VIX_Price_zscore_60d__minus__GD_GeneralDynamics_zscore_60d", "VIX_Price_zscore_60d__minus__CLX_Clorox_zscore_60d", "VIX_Price_zscore_60d__zrel__CLX_Clorox_zscore_60d", "vix_zscore_10d__macross__RTX_Raytheon_zscore_60d", "vix_vol_of_vol_10d__minus__XLU_Util_ret_5d", "Nikkei_Japan_vol_20d", "NFCI_ret_5d__div__NFCI_vol_20d", "VIX_Price_zscore_60d__div__vix_vol_of_vol_10d", "VIX_Price_zscore_60d__zrel__GD_GeneralDynamics_zscore_60d", "vix_vol_of_vol_10d__prod__vix_level", "vix_zscore_10d__zrel__EWP_Spain_zscore_60d"], "is_new": false}, {"model_id": "v2_h7_NORMAL_RandomForest_N19", "algo": "RandomForest", "regime": "NORMAL", "horizon": 7, "n_features": 19, "F1_dir": 0.5768, "F1_UP_FORT": 0.3611, "F1_DOWN_FORT": 0.4731, "train_start": "2001-02-06", "sampler": "SMOTEENN", "best_params": "{}", "features": ["NFCI_ret_5d__minus__NFCI_vol_20d", "VIX_Price_zscore_60d__div__heston_var_ev_h1", "VIX_Price_zscore_60d__div__NFCI_vol_20d", "VIX_Price_zscore_60d__prod__Nikkei_Japan_vol_20d", "GD_GeneralDynamics_zscore_60d__ret5x__COST_zscore_60d", "vix_vol_of_vol_10d__zrel__XLU_Util_ret_5d", "HD_ret_20d", "VIX_Price_zscore_60d__minus__GD_GeneralDynamics_zscore_60d", "VIX_Price_zscore_60d__minus__CLX_Clorox_zscore_60d", "VIX_Price_zscore_60d__zrel__CLX_Clorox_zscore_60d", "vix_zscore_10d__macross__RTX_Raytheon_zscore_60d", "vix_vol_of_vol_10d__minus__XLU_Util_ret_5d", "Nikkei_Japan_vol_20d", "NFCI_ret_5d__div__NFCI_vol_20d", "VIX_Price_zscore_60d__div__vix_vol_of_vol_10d", "VIX_Price_zscore_60d__zrel__GD_GeneralDynamics_zscore_60d", "vix_vol_of_vol_10d__prod__vix_level", "vix_zscore_10d__zrel__EWP_Spain_zscore_60d", "IYR_US_REIT2_zscore_60d"], "is_new": false}, {"model_id": "v2_h7_NORMAL_RandomForest_N20", "algo": "RandomForest", "regime": "NORMAL", "horizon": 7, "n_features": 20, "F1_dir": 0.5215, "F1_UP_FORT": 0.3859, "F1_DOWN_FORT": 0.47, "train_start": "2001-02-06", "sampler": "SMOTEENN", "best_params": "{}", "features": ["NFCI_ret_5d__minus__NFCI_vol_20d", "VIX_Price_zscore_60d__div__heston_var_ev_h1", "VIX_Price_zscore_60d__div__NFCI_vol_20d", "VIX_Price_zscore_60d__prod__Nikkei_Japan_vol_20d", "GD_GeneralDynamics_zscore_60d__ret5x__COST_zscore_60d", "vix_vol_of_vol_10d__zrel__XLU_Util_ret_5d", "HD_ret_20d", "VIX_Price_zscore_60d__minus__GD_GeneralDynamics_zscore_60d", "VIX_Price_zscore_60d__minus__CLX_Clorox_zscore_60d", "VIX_Price_zscore_60d__zrel__CLX_Clorox_zscore_60d", "vix_zscore_10d__macross__RTX_Raytheon_zscore_60d", "vix_vol_of_vol_10d__minus__XLU_Util_ret_5d", "Nikkei_Japan_vol_20d", "NFCI_ret_5d__div__NFCI_vol_20d", "VIX_Price_zscore_60d__div__vix_vol_of_vol_10d", "VIX_Price_zscore_60d__zrel__GD_GeneralDynamics_zscore_60d", "vix_vol_of_vol_10d__prod__vix_level", "vix_zscore_10d__zrel__EWP_Spain_zscore_60d", "IYR_US_REIT2_zscore_60d", "heston_var_ev_h1__div__NFCI_vol_20d"], "is_new": false}, {"model_id": "v2_h7_NORMAL_LogisticRegression_N6", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 7, "n_features": 6, "F1_dir": 0.5018, "F1_UP_FORT": 0.32, "F1_DOWN_FORT": 0.4431, "train_start": "2001-02-06", "sampler": "SMOTEENN", "best_params": "{}", "features": ["NFCI_ret_5d__minus__NFCI_vol_20d", "VIX_Price_zscore_60d__div__heston_var_ev_h1", "VIX_Price_zscore_60d__div__NFCI_vol_20d", "VIX_Price_zscore_60d__prod__Nikkei_Japan_vol_20d", "GD_GeneralDynamics_zscore_60d__ret5x__COST_zscore_60d", "vix_vol_of_vol_10d__zrel__XLU_Util_ret_5d"], "is_new": false}, {"model_id": "v2_h7_NORMAL_LogisticRegression_N7", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 7, "n_features": 7, "F1_dir": 0.5091, "F1_UP_FORT": 0.3641, "F1_DOWN_FORT": 0.4596, "train_start": "2001-02-06", "sampler": "SMOTEENN", "best_params": "{}", "features": ["NFCI_ret_5d__minus__NFCI_vol_20d", "VIX_Price_zscore_60d__div__heston_var_ev_h1", "VIX_Price_zscore_60d__div__NFCI_vol_20d", "VIX_Price_zscore_60d__prod__Nikkei_Japan_vol_20d", "GD_GeneralDynamics_zscore_60d__ret5x__COST_zscore_60d", "vix_vol_of_vol_10d__zrel__XLU_Util_ret_5d", "HD_ret_20d"], "is_new": false}, {"model_id": "v2_h7_NORMAL_LogisticRegression_N8", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 7, "n_features": 8, "F1_dir": 0.5146, "F1_UP_FORT": 0.3697, "F1_DOWN_FORT": 0.429, "train_start": "2001-02-06", "sampler": "SMOTEENN", "best_params": "{}", "features": ["NFCI_ret_5d__minus__NFCI_vol_20d", "VIX_Price_zscore_60d__div__heston_var_ev_h1", "VIX_Price_zscore_60d__div__NFCI_vol_20d", "VIX_Price_zscore_60d__prod__Nikkei_Japan_vol_20d", "GD_GeneralDynamics_zscore_60d__ret5x__COST_zscore_60d", "vix_vol_of_vol_10d__zrel__XLU_Util_ret_5d", "HD_ret_20d", "VIX_Price_zscore_60d__minus__GD_GeneralDynamics_zscore_60d"], "is_new": false}, {"model_id": "v2_h7_NORMAL_LogisticRegression_N9", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 7, "n_features": 9, "F1_dir": 0.5235, "F1_UP_FORT": 0.3186, "F1_DOWN_FORT": 0.4471, "train_start": "2001-02-06", "sampler": "SMOTEENN", "best_params": "{}", "features": ["NFCI_ret_5d__minus__NFCI_vol_20d", "VIX_Price_zscore_60d__div__heston_var_ev_h1", "VIX_Price_zscore_60d__div__NFCI_vol_20d", "VIX_Price_zscore_60d__prod__Nikkei_Japan_vol_20d", "GD_GeneralDynamics_zscore_60d__ret5x__COST_zscore_60d", "vix_vol_of_vol_10d__zrel__XLU_Util_ret_5d", "HD_ret_20d", "VIX_Price_zscore_60d__minus__GD_GeneralDynamics_zscore_60d", "VIX_Price_zscore_60d__minus__CLX_Clorox_zscore_60d"], "is_new": false}, {"model_id": "v2_h7_NORMAL_LogisticRegression_N10", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 7, "n_features": 10, "F1_dir": 0.5486, "F1_UP_FORT": 0.3718, "F1_DOWN_FORT": 0.4201, "train_start": "2001-02-06", "sampler": "SMOTEENN", "best_params": "{}", "features": ["NFCI_ret_5d__minus__NFCI_vol_20d", "VIX_Price_zscore_60d__div__heston_var_ev_h1", "VIX_Price_zscore_60d__div__NFCI_vol_20d", "VIX_Price_zscore_60d__prod__Nikkei_Japan_vol_20d", "GD_GeneralDynamics_zscore_60d__ret5x__COST_zscore_60d", "vix_vol_of_vol_10d__zrel__XLU_Util_ret_5d", "HD_ret_20d", "VIX_Price_zscore_60d__minus__GD_GeneralDynamics_zscore_60d", "VIX_Price_zscore_60d__minus__CLX_Clorox_zscore_60d", "VIX_Price_zscore_60d__zrel__CLX_Clorox_zscore_60d"], "is_new": false}, {"model_id": "v2_h7_NORMAL_LogisticRegression_N11", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 7, "n_features": 11, "F1_dir": 0.5391, "F1_UP_FORT": 0.3301, "F1_DOWN_FORT": 0.4334, "train_start": "2001-02-06", "sampler": "SMOTEENN", "best_params": "{}", "features": ["NFCI_ret_5d__minus__NFCI_vol_20d", "VIX_Price_zscore_60d__div__heston_var_ev_h1", "VIX_Price_zscore_60d__div__NFCI_vol_20d", "VIX_Price_zscore_60d__prod__Nikkei_Japan_vol_20d", "GD_GeneralDynamics_zscore_60d__ret5x__COST_zscore_60d", "vix_vol_of_vol_10d__zrel__XLU_Util_ret_5d", "HD_ret_20d", "VIX_Price_zscore_60d__minus__GD_GeneralDynamics_zscore_60d", "VIX_Price_zscore_60d__minus__CLX_Clorox_zscore_60d", "VIX_Price_zscore_60d__zrel__CLX_Clorox_zscore_60d", "vix_zscore_10d__macross__RTX_Raytheon_zscore_60d"], "is_new": false}, {"model_id": "v2_h7_NORMAL_LogisticRegression_N12", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 7, "n_features": 12, "F1_dir": 0.5133, "F1_UP_FORT": 0.3212, "F1_DOWN_FORT": 0.4023, "train_start": "2001-02-06", "sampler": "SMOTEENN", "best_params": "{}", "features": ["NFCI_ret_5d__minus__NFCI_vol_20d", "VIX_Price_zscore_60d__div__heston_var_ev_h1", "VIX_Price_zscore_60d__div__NFCI_vol_20d", "VIX_Price_zscore_60d__prod__Nikkei_Japan_vol_20d", "GD_GeneralDynamics_zscore_60d__ret5x__COST_zscore_60d", "vix_vol_of_vol_10d__zrel__XLU_Util_ret_5d", "HD_ret_20d", "VIX_Price_zscore_60d__minus__GD_GeneralDynamics_zscore_60d", "VIX_Price_zscore_60d__minus__CLX_Clorox_zscore_60d", "VIX_Price_zscore_60d__zrel__CLX_Clorox_zscore_60d", "vix_zscore_10d__macross__RTX_Raytheon_zscore_60d", "vix_vol_of_vol_10d__minus__XLU_Util_ret_5d"], "is_new": false}, {"model_id": "v2_h7_NORMAL_LogisticRegression_N13", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 7, "n_features": 13, "F1_dir": 0.5206, "F1_UP_FORT": 0.3135, "F1_DOWN_FORT": 0.375, "train_start": "2001-02-06", "sampler": "SMOTEENN", "best_params": "{}", "features": ["NFCI_ret_5d__minus__NFCI_vol_20d", "VIX_Price_zscore_60d__div__heston_var_ev_h1", "VIX_Price_zscore_60d__div__NFCI_vol_20d", "VIX_Price_zscore_60d__prod__Nikkei_Japan_vol_20d", "GD_GeneralDynamics_zscore_60d__ret5x__COST_zscore_60d", "vix_vol_of_vol_10d__zrel__XLU_Util_ret_5d", "HD_ret_20d", "VIX_Price_zscore_60d__minus__GD_GeneralDynamics_zscore_60d", "VIX_Price_zscore_60d__minus__CLX_Clorox_zscore_60d", "VIX_Price_zscore_60d__zrel__CLX_Clorox_zscore_60d", "vix_zscore_10d__macross__RTX_Raytheon_zscore_60d", "vix_vol_of_vol_10d__minus__XLU_Util_ret_5d", "Nikkei_Japan_vol_20d"], "is_new": false}, {"model_id": "v2_h7_NORMAL_LogisticRegression_N14", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 7, "n_features": 14, "F1_dir": 0.526, "F1_UP_FORT": 0.3243, "F1_DOWN_FORT": 0.3846, "train_start": "2001-02-06", "sampler": "SMOTEENN", "best_params": "{}", "features": ["NFCI_ret_5d__minus__NFCI_vol_20d", "VIX_Price_zscore_60d__div__heston_var_ev_h1", "VIX_Price_zscore_60d__div__NFCI_vol_20d", "VIX_Price_zscore_60d__prod__Nikkei_Japan_vol_20d", "GD_GeneralDynamics_zscore_60d__ret5x__COST_zscore_60d", "vix_vol_of_vol_10d__zrel__XLU_Util_ret_5d", "HD_ret_20d", "VIX_Price_zscore_60d__minus__GD_GeneralDynamics_zscore_60d", "VIX_Price_zscore_60d__minus__CLX_Clorox_zscore_60d", "VIX_Price_zscore_60d__zrel__CLX_Clorox_zscore_60d", "vix_zscore_10d__macross__RTX_Raytheon_zscore_60d", "vix_vol_of_vol_10d__minus__XLU_Util_ret_5d", "Nikkei_Japan_vol_20d", "NFCI_ret_5d__div__NFCI_vol_20d"], "is_new": false}, {"model_id": "v2_h7_NORMAL_LogisticRegression_N15", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 7, "n_features": 15, "F1_dir": 0.523, "F1_UP_FORT": 0.3145, "F1_DOWN_FORT": 0.3846, "train_start": "2001-02-06", "sampler": "SMOTEENN", "best_params": "{}", "features": ["NFCI_ret_5d__minus__NFCI_vol_20d", "VIX_Price_zscore_60d__div__heston_var_ev_h1", "VIX_Price_zscore_60d__div__NFCI_vol_20d", "VIX_Price_zscore_60d__prod__Nikkei_Japan_vol_20d", "GD_GeneralDynamics_zscore_60d__ret5x__COST_zscore_60d", "vix_vol_of_vol_10d__zrel__XLU_Util_ret_5d", "HD_ret_20d", "VIX_Price_zscore_60d__minus__GD_GeneralDynamics_zscore_60d", "VIX_Price_zscore_60d__minus__CLX_Clorox_zscore_60d", "VIX_Price_zscore_60d__zrel__CLX_Clorox_zscore_60d", "vix_zscore_10d__macross__RTX_Raytheon_zscore_60d", "vix_vol_of_vol_10d__minus__XLU_Util_ret_5d", "Nikkei_Japan_vol_20d", "NFCI_ret_5d__div__NFCI_vol_20d", "VIX_Price_zscore_60d__div__vix_vol_of_vol_10d"], "is_new": false}, {"model_id": "v2_h7_NORMAL_LogisticRegression_N16", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 7, "n_features": 16, "F1_dir": 0.543, "F1_UP_FORT": 0.3137, "F1_DOWN_FORT": 0.3979, "train_start": "2001-02-06", "sampler": "SMOTEENN", "best_params": "{}", "features": ["NFCI_ret_5d__minus__NFCI_vol_20d", "VIX_Price_zscore_60d__div__heston_var_ev_h1", "VIX_Price_zscore_60d__div__NFCI_vol_20d", "VIX_Price_zscore_60d__prod__Nikkei_Japan_vol_20d", "GD_GeneralDynamics_zscore_60d__ret5x__COST_zscore_60d", "vix_vol_of_vol_10d__zrel__XLU_Util_ret_5d", "HD_ret_20d", "VIX_Price_zscore_60d__minus__GD_GeneralDynamics_zscore_60d", "VIX_Price_zscore_60d__minus__CLX_Clorox_zscore_60d", "VIX_Price_zscore_60d__zrel__CLX_Clorox_zscore_60d", "vix_zscore_10d__macross__RTX_Raytheon_zscore_60d", "vix_vol_of_vol_10d__minus__XLU_Util_ret_5d", "Nikkei_Japan_vol_20d", "NFCI_ret_5d__div__NFCI_vol_20d", "VIX_Price_zscore_60d__div__vix_vol_of_vol_10d", "VIX_Price_zscore_60d__zrel__GD_GeneralDynamics_zscore_60d"], "is_new": false}, {"model_id": "v2_h7_NORMAL_LogisticRegression_N17", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 7, "n_features": 17, "F1_dir": 0.5434, "F1_UP_FORT": 0.3325, "F1_DOWN_FORT": 0.4076, "train_start": "2001-02-06", "sampler": "SMOTEENN", "best_params": "{}", "features": ["NFCI_ret_5d__minus__NFCI_vol_20d", "VIX_Price_zscore_60d__div__heston_var_ev_h1", "VIX_Price_zscore_60d__div__NFCI_vol_20d", "VIX_Price_zscore_60d__prod__Nikkei_Japan_vol_20d", "GD_GeneralDynamics_zscore_60d__ret5x__COST_zscore_60d", "vix_vol_of_vol_10d__zrel__XLU_Util_ret_5d", "HD_ret_20d", "VIX_Price_zscore_60d__minus__GD_GeneralDynamics_zscore_60d", "VIX_Price_zscore_60d__minus__CLX_Clorox_zscore_60d", "VIX_Price_zscore_60d__zrel__CLX_Clorox_zscore_60d", "vix_zscore_10d__macross__RTX_Raytheon_zscore_60d", "vix_vol_of_vol_10d__minus__XLU_Util_ret_5d", "Nikkei_Japan_vol_20d", "NFCI_ret_5d__div__NFCI_vol_20d", "VIX_Price_zscore_60d__div__vix_vol_of_vol_10d", "VIX_Price_zscore_60d__zrel__GD_GeneralDynamics_zscore_60d", "vix_vol_of_vol_10d__prod__vix_level"], "is_new": false}, {"model_id": "v2_h7_NORMAL_LogisticRegression_N18", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 7, "n_features": 18, "F1_dir": 0.5772, "F1_UP_FORT": 0.3033, "F1_DOWN_FORT": 0.4405, "train_start": "2001-02-06", "sampler": "SMOTEENN", "best_params": "{}", "features": ["NFCI_ret_5d__minus__NFCI_vol_20d", "VIX_Price_zscore_60d__div__heston_var_ev_h1", "VIX_Price_zscore_60d__div__NFCI_vol_20d", "VIX_Price_zscore_60d__prod__Nikkei_Japan_vol_20d", "GD_GeneralDynamics_zscore_60d__ret5x__COST_zscore_60d", "vix_vol_of_vol_10d__zrel__XLU_Util_ret_5d", "HD_ret_20d", "VIX_Price_zscore_60d__minus__GD_GeneralDynamics_zscore_60d", "VIX_Price_zscore_60d__minus__CLX_Clorox_zscore_60d", "VIX_Price_zscore_60d__zrel__CLX_Clorox_zscore_60d", "vix_zscore_10d__macross__RTX_Raytheon_zscore_60d", "vix_vol_of_vol_10d__minus__XLU_Util_ret_5d", "Nikkei_Japan_vol_20d", "NFCI_ret_5d__div__NFCI_vol_20d", "VIX_Price_zscore_60d__div__vix_vol_of_vol_10d", "VIX_Price_zscore_60d__zrel__GD_GeneralDynamics_zscore_60d", "vix_vol_of_vol_10d__prod__vix_level", "vix_zscore_10d__zrel__EWP_Spain_zscore_60d"], "is_new": false}, {"model_id": "v2_h7_NORMAL_LogisticRegression_N19", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 7, "n_features": 19, "F1_dir": 0.5718, "F1_UP_FORT": 0.3166, "F1_DOWN_FORT": 0.419, "train_start": "2001-02-06", "sampler": "SMOTEENN", "best_params": "{}", "features": ["NFCI_ret_5d__minus__NFCI_vol_20d", "VIX_Price_zscore_60d__div__heston_var_ev_h1", "VIX_Price_zscore_60d__div__NFCI_vol_20d", "VIX_Price_zscore_60d__prod__Nikkei_Japan_vol_20d", "GD_GeneralDynamics_zscore_60d__ret5x__COST_zscore_60d", "vix_vol_of_vol_10d__zrel__XLU_Util_ret_5d", "HD_ret_20d", "VIX_Price_zscore_60d__minus__GD_GeneralDynamics_zscore_60d", "VIX_Price_zscore_60d__minus__CLX_Clorox_zscore_60d", "VIX_Price_zscore_60d__zrel__CLX_Clorox_zscore_60d", "vix_zscore_10d__macross__RTX_Raytheon_zscore_60d", "vix_vol_of_vol_10d__minus__XLU_Util_ret_5d", "Nikkei_Japan_vol_20d", "NFCI_ret_5d__div__NFCI_vol_20d", "VIX_Price_zscore_60d__div__vix_vol_of_vol_10d", "VIX_Price_zscore_60d__zrel__GD_GeneralDynamics_zscore_60d", "vix_vol_of_vol_10d__prod__vix_level", "vix_zscore_10d__zrel__EWP_Spain_zscore_60d", "IYR_US_REIT2_zscore_60d"], "is_new": false}, {"model_id": "v2_h7_NORMAL_LogisticRegression_N20", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 7, "n_features": 20, "F1_dir": 0.514, "F1_UP_FORT": 0.1864, "F1_DOWN_FORT": 0.4234, "train_start": "2001-02-06", "sampler": "SMOTEENN", "best_params": "{}", "features": ["NFCI_ret_5d__minus__NFCI_vol_20d", "VIX_Price_zscore_60d__div__heston_var_ev_h1", "VIX_Price_zscore_60d__div__NFCI_vol_20d", "VIX_Price_zscore_60d__prod__Nikkei_Japan_vol_20d", "GD_GeneralDynamics_zscore_60d__ret5x__COST_zscore_60d", "vix_vol_of_vol_10d__zrel__XLU_Util_ret_5d", "HD_ret_20d", "VIX_Price_zscore_60d__minus__GD_GeneralDynamics_zscore_60d", "VIX_Price_zscore_60d__minus__CLX_Clorox_zscore_60d", "VIX_Price_zscore_60d__zrel__CLX_Clorox_zscore_60d", "vix_zscore_10d__macross__RTX_Raytheon_zscore_60d", "vix_vol_of_vol_10d__minus__XLU_Util_ret_5d", "Nikkei_Japan_vol_20d", "NFCI_ret_5d__div__NFCI_vol_20d", "VIX_Price_zscore_60d__div__vix_vol_of_vol_10d", "VIX_Price_zscore_60d__zrel__GD_GeneralDynamics_zscore_60d", "vix_vol_of_vol_10d__prod__vix_level", "vix_zscore_10d__zrel__EWP_Spain_zscore_60d", "IYR_US_REIT2_zscore_60d", "heston_var_ev_h1__div__NFCI_vol_20d"], "is_new": false}, {"model_id": "v2_h7_NORMAL_RandomForest_Optuna_N18", "algo": "RandomForest", "regime": "NORMAL", "horizon": 7, "n_features": 18, "F1_dir": 0.6292, "F1_UP_FORT": 0.489, "F1_DOWN_FORT": 0.4629, "train_start": "2001-02-06", "sampler": "SMOTEENN", "best_params": "{}", "features": ["NFCI_ret_5d__minus__NFCI_vol_20d", "VIX_Price_zscore_60d__div__heston_var_ev_h1", "VIX_Price_zscore_60d__div__NFCI_vol_20d", "VIX_Price_zscore_60d__prod__Nikkei_Japan_vol_20d", "GD_GeneralDynamics_zscore_60d__ret5x__COST_zscore_60d", "vix_vol_of_vol_10d__zrel__XLU_Util_ret_5d", "HD_ret_20d", "VIX_Price_zscore_60d__minus__GD_GeneralDynamics_zscore_60d", "VIX_Price_zscore_60d__minus__CLX_Clorox_zscore_60d", "VIX_Price_zscore_60d__zrel__CLX_Clorox_zscore_60d", "vix_zscore_10d__macross__RTX_Raytheon_zscore_60d", "vix_vol_of_vol_10d__minus__XLU_Util_ret_5d", "Nikkei_Japan_vol_20d", "NFCI_ret_5d__div__NFCI_vol_20d", "VIX_Price_zscore_60d__div__vix_vol_of_vol_10d", "VIX_Price_zscore_60d__zrel__GD_GeneralDynamics_zscore_60d", "vix_vol_of_vol_10d__prod__vix_level", "vix_zscore_10d__zrel__EWP_Spain_zscore_60d"], "is_new": false}, {"model_id": "v2_h7_STRESS_XGBoost_N7", "algo": "XGBoost", "regime": "STRESS", "horizon": 7, "n_features": 7, "F1_dir": 0.5789, "F1_UP_FORT": 0.323, "F1_DOWN_FORT": 0.4074, "train_start": "2000-11-15", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["US3M_Rate_vol_20d", "AMD_zscore_60d__ret5x__XLU_Util_ret_5d", "DE_Deere_ret_5d", "VIX_Price_zscore_60d__macross__AMD_zscore_60d", "VIX_Price_zscore_60d__prod__JNJ_ret_20d", "JNJ_ret_20d__macross__vix_vs_ma5", "US30Y_Rate_ret_20d"], "is_new": false}, {"model_id": "v2_h7_STRESS_XGBoost_N8", "algo": "XGBoost", "regime": "STRESS", "horizon": 7, "n_features": 8, "F1_dir": 0.5675, "F1_UP_FORT": 0.3356, "F1_DOWN_FORT": 0.4091, "train_start": "2000-11-15", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["US3M_Rate_vol_20d", "AMD_zscore_60d__ret5x__XLU_Util_ret_5d", "DE_Deere_ret_5d", "VIX_Price_zscore_60d__macross__AMD_zscore_60d", "VIX_Price_zscore_60d__prod__JNJ_ret_20d", "JNJ_ret_20d__macross__vix_vs_ma5", "US30Y_Rate_ret_20d", "VRP_ma5"], "is_new": false}, {"model_id": "v2_h7_STRESS_XGBoost_N9", "algo": "XGBoost", "regime": "STRESS", "horizon": 7, "n_features": 9, "F1_dir": 0.5657, "F1_UP_FORT": 0.2714, "F1_DOWN_FORT": 0.4394, "train_start": "2000-11-15", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["US3M_Rate_vol_20d", "AMD_zscore_60d__ret5x__XLU_Util_ret_5d", "DE_Deere_ret_5d", "VIX_Price_zscore_60d__macross__AMD_zscore_60d", "VIX_Price_zscore_60d__prod__JNJ_ret_20d", "JNJ_ret_20d__macross__vix_vs_ma5", "US30Y_Rate_ret_20d", "VRP_ma5", "DHR_vol_20d"], "is_new": false}, {"model_id": "v2_h7_STRESS_XGBoost_N10", "algo": "XGBoost", "regime": "STRESS", "horizon": 7, "n_features": 10, "F1_dir": 0.549, "F1_UP_FORT": 0.2329, "F1_DOWN_FORT": 0.4296, "train_start": "2000-11-15", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["US3M_Rate_vol_20d", "AMD_zscore_60d__ret5x__XLU_Util_ret_5d", "DE_Deere_ret_5d", "VIX_Price_zscore_60d__macross__AMD_zscore_60d", "VIX_Price_zscore_60d__prod__JNJ_ret_20d", "JNJ_ret_20d__macross__vix_vs_ma5", "US30Y_Rate_ret_20d", "VRP_ma5", "DHR_vol_20d", "VIX_Price_zscore_60d__ret5x__JNJ_ret_20d"], "is_new": false}, {"model_id": "v2_h7_STRESS_XGBoost_N11", "algo": "XGBoost", "regime": "STRESS", "horizon": 7, "n_features": 11, "F1_dir": 0.5517, "F1_UP_FORT": 0.1778, "F1_DOWN_FORT": 0.4351, "train_start": "2000-11-15", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["US3M_Rate_vol_20d", "AMD_zscore_60d__ret5x__XLU_Util_ret_5d", "DE_Deere_ret_5d", "VIX_Price_zscore_60d__macross__AMD_zscore_60d", "VIX_Price_zscore_60d__prod__JNJ_ret_20d", "JNJ_ret_20d__macross__vix_vs_ma5", "US30Y_Rate_ret_20d", "VRP_ma5", "DHR_vol_20d", "VIX_Price_zscore_60d__ret5x__JNJ_ret_20d", "vix_mean_abs_ret_5d"], "is_new": false}, {"model_id": "v2_h7_STRESS_XGBoost_N12", "algo": "XGBoost", "regime": "STRESS", "horizon": 7, "n_features": 12, "F1_dir": 0.5148, "F1_UP_FORT": 0.1481, "F1_DOWN_FORT": 0.4521, "train_start": "2000-11-15", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["US3M_Rate_vol_20d", "AMD_zscore_60d__ret5x__XLU_Util_ret_5d", "DE_Deere_ret_5d", "VIX_Price_zscore_60d__macross__AMD_zscore_60d", "VIX_Price_zscore_60d__prod__JNJ_ret_20d", "JNJ_ret_20d__macross__vix_vs_ma5", "US30Y_Rate_ret_20d", "VRP_ma5", "DHR_vol_20d", "VIX_Price_zscore_60d__ret5x__JNJ_ret_20d", "vix_mean_abs_ret_5d", "DOW_Price_zscore_60d"], "is_new": false}, {"model_id": "v2_h7_STRESS_XGBoost_N13", "algo": "XGBoost", "regime": "STRESS", "horizon": 7, "n_features": 13, "F1_dir": 0.5293, "F1_UP_FORT": 0.1356, "F1_DOWN_FORT": 0.4514, "train_start": "2000-11-15", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["US3M_Rate_vol_20d", "AMD_zscore_60d__ret5x__XLU_Util_ret_5d", "DE_Deere_ret_5d", "VIX_Price_zscore_60d__macross__AMD_zscore_60d", "VIX_Price_zscore_60d__prod__JNJ_ret_20d", "JNJ_ret_20d__macross__vix_vs_ma5", "US30Y_Rate_ret_20d", "VRP_ma5", "DHR_vol_20d", "VIX_Price_zscore_60d__ret5x__JNJ_ret_20d", "vix_mean_abs_ret_5d", "DOW_Price_zscore_60d", "BAX_BankBoston_ret_20d__div__XLU_Util_ret_5d"], "is_new": false}, {"model_id": "v2_h7_STRESS_XGBoost_N14", "algo": "XGBoost", "regime": "STRESS", "horizon": 7, "n_features": 14, "F1_dir": 0.5623, "F1_UP_FORT": 0.15, "F1_DOWN_FORT": 0.5017, "train_start": "2000-11-15", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["US3M_Rate_vol_20d", "AMD_zscore_60d__ret5x__XLU_Util_ret_5d", "DE_Deere_ret_5d", "VIX_Price_zscore_60d__macross__AMD_zscore_60d", "VIX_Price_zscore_60d__prod__JNJ_ret_20d", "JNJ_ret_20d__macross__vix_vs_ma5", "US30Y_Rate_ret_20d", "VRP_ma5", "DHR_vol_20d", "VIX_Price_zscore_60d__ret5x__JNJ_ret_20d", "vix_mean_abs_ret_5d", "DOW_Price_zscore_60d", "BAX_BankBoston_ret_20d__div__XLU_Util_ret_5d", "WMT_zscore_60d__div__BBY_BestBuy_zscore_60d"], "is_new": false}, {"model_id": "v2_h7_STRESS_XGBoost_N15", "algo": "XGBoost", "regime": "STRESS", "horizon": 7, "n_features": 15, "F1_dir": 0.5101, "F1_UP_FORT": 0.1311, "F1_DOWN_FORT": 0.4507, "train_start": "2000-11-15", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["US3M_Rate_vol_20d", "AMD_zscore_60d__ret5x__XLU_Util_ret_5d", "DE_Deere_ret_5d", "VIX_Price_zscore_60d__macross__AMD_zscore_60d", "VIX_Price_zscore_60d__prod__JNJ_ret_20d", "JNJ_ret_20d__macross__vix_vs_ma5", "US30Y_Rate_ret_20d", "VRP_ma5", "DHR_vol_20d", "VIX_Price_zscore_60d__ret5x__JNJ_ret_20d", "vix_mean_abs_ret_5d", "DOW_Price_zscore_60d", "BAX_BankBoston_ret_20d__div__XLU_Util_ret_5d", "WMT_zscore_60d__div__BBY_BestBuy_zscore_60d", "FedFunds_zscore_60d"], "is_new": false}, {"model_id": "v2_h7_STRESS_LightGBM_N5", "algo": "LightGBM", "regime": "STRESS", "horizon": 7, "n_features": 5, "F1_dir": 0.5026, "F1_UP_FORT": 0.3038, "F1_DOWN_FORT": 0.4016, "train_start": "2000-11-15", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["US3M_Rate_vol_20d", "AMD_zscore_60d__ret5x__XLU_Util_ret_5d", "DE_Deere_ret_5d", "VIX_Price_zscore_60d__macross__AMD_zscore_60d", "VIX_Price_zscore_60d__prod__JNJ_ret_20d"], "is_new": false}, {"model_id": "v2_h7_STRESS_LightGBM_N6", "algo": "LightGBM", "regime": "STRESS", "horizon": 7, "n_features": 6, "F1_dir": 0.5234, "F1_UP_FORT": 0.2793, "F1_DOWN_FORT": 0.3419, "train_start": "2000-11-15", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["US3M_Rate_vol_20d", "AMD_zscore_60d__ret5x__XLU_Util_ret_5d", "DE_Deere_ret_5d", "VIX_Price_zscore_60d__macross__AMD_zscore_60d", "VIX_Price_zscore_60d__prod__JNJ_ret_20d", "JNJ_ret_20d__macross__vix_vs_ma5"], "is_new": false}, {"model_id": "v2_h7_STRESS_LightGBM_N7", "algo": "LightGBM", "regime": "STRESS", "horizon": 7, "n_features": 7, "F1_dir": 0.5946, "F1_UP_FORT": 0.3158, "F1_DOWN_FORT": 0.4312, "train_start": "2000-11-15", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["US3M_Rate_vol_20d", "AMD_zscore_60d__ret5x__XLU_Util_ret_5d", "DE_Deere_ret_5d", "VIX_Price_zscore_60d__macross__AMD_zscore_60d", "VIX_Price_zscore_60d__prod__JNJ_ret_20d", "JNJ_ret_20d__macross__vix_vs_ma5", "US30Y_Rate_ret_20d"], "is_new": false}, {"model_id": "v2_h7_STRESS_LightGBM_N8", "algo": "LightGBM", "regime": "STRESS", "horizon": 7, "n_features": 8, "F1_dir": 0.6185, "F1_UP_FORT": 0.3265, "F1_DOWN_FORT": 0.4125, "train_start": "2000-11-15", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["US3M_Rate_vol_20d", "AMD_zscore_60d__ret5x__XLU_Util_ret_5d", "DE_Deere_ret_5d", "VIX_Price_zscore_60d__macross__AMD_zscore_60d", "VIX_Price_zscore_60d__prod__JNJ_ret_20d", "JNJ_ret_20d__macross__vix_vs_ma5", "US30Y_Rate_ret_20d", "VRP_ma5"], "is_new": false}, {"model_id": "v2_h7_STRESS_LightGBM_N9", "algo": "LightGBM", "regime": "STRESS", "horizon": 7, "n_features": 9, "F1_dir": 0.6175, "F1_UP_FORT": 0.2295, "F1_DOWN_FORT": 0.3636, "train_start": "2000-11-15", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["US3M_Rate_vol_20d", "AMD_zscore_60d__ret5x__XLU_Util_ret_5d", "DE_Deere_ret_5d", "VIX_Price_zscore_60d__macross__AMD_zscore_60d", "VIX_Price_zscore_60d__prod__JNJ_ret_20d", "JNJ_ret_20d__macross__vix_vs_ma5", "US30Y_Rate_ret_20d", "VRP_ma5", "DHR_vol_20d"], "is_new": false}, {"model_id": "v2_h7_STRESS_LightGBM_N10", "algo": "LightGBM", "regime": "STRESS", "horizon": 7, "n_features": 10, "F1_dir": 0.6132, "F1_UP_FORT": 0.2927, "F1_DOWN_FORT": 0.3534, "train_start": "2000-11-15", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["US3M_Rate_vol_20d", "AMD_zscore_60d__ret5x__XLU_Util_ret_5d", "DE_Deere_ret_5d", "VIX_Price_zscore_60d__macross__AMD_zscore_60d", "VIX_Price_zscore_60d__prod__JNJ_ret_20d", "JNJ_ret_20d__macross__vix_vs_ma5", "US30Y_Rate_ret_20d", "VRP_ma5", "DHR_vol_20d", "VIX_Price_zscore_60d__ret5x__JNJ_ret_20d"], "is_new": false}, {"model_id": "v2_h7_STRESS_LightGBM_N11", "algo": "LightGBM", "regime": "STRESS", "horizon": 7, "n_features": 11, "F1_dir": 0.5862, "F1_UP_FORT": 0.194, "F1_DOWN_FORT": 0.4615, "train_start": "2000-11-15", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["US3M_Rate_vol_20d", "AMD_zscore_60d__ret5x__XLU_Util_ret_5d", "DE_Deere_ret_5d", "VIX_Price_zscore_60d__macross__AMD_zscore_60d", "VIX_Price_zscore_60d__prod__JNJ_ret_20d", "JNJ_ret_20d__macross__vix_vs_ma5", "US30Y_Rate_ret_20d", "VRP_ma5", "DHR_vol_20d", "VIX_Price_zscore_60d__ret5x__JNJ_ret_20d", "vix_mean_abs_ret_5d"], "is_new": false}, {"model_id": "v2_h7_STRESS_LightGBM_N12", "algo": "LightGBM", "regime": "STRESS", "horizon": 7, "n_features": 12, "F1_dir": 0.5793, "F1_UP_FORT": 0.1835, "F1_DOWN_FORT": 0.4784, "train_start": "2000-11-15", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["US3M_Rate_vol_20d", "AMD_zscore_60d__ret5x__XLU_Util_ret_5d", "DE_Deere_ret_5d", "VIX_Price_zscore_60d__macross__AMD_zscore_60d", "VIX_Price_zscore_60d__prod__JNJ_ret_20d", "JNJ_ret_20d__macross__vix_vs_ma5", "US30Y_Rate_ret_20d", "VRP_ma5", "DHR_vol_20d", "VIX_Price_zscore_60d__ret5x__JNJ_ret_20d", "vix_mean_abs_ret_5d", "DOW_Price_zscore_60d"], "is_new": false}, {"model_id": "v2_h7_STRESS_LightGBM_N13", "algo": "LightGBM", "regime": "STRESS", "horizon": 7, "n_features": 13, "F1_dir": 0.562, "F1_UP_FORT": 0.1724, "F1_DOWN_FORT": 0.5034, "train_start": "2000-11-15", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["US3M_Rate_vol_20d", "AMD_zscore_60d__ret5x__XLU_Util_ret_5d", "DE_Deere_ret_5d", "VIX_Price_zscore_60d__macross__AMD_zscore_60d", "VIX_Price_zscore_60d__prod__JNJ_ret_20d", "JNJ_ret_20d__macross__vix_vs_ma5", "US30Y_Rate_ret_20d", "VRP_ma5", "DHR_vol_20d", "VIX_Price_zscore_60d__ret5x__JNJ_ret_20d", "vix_mean_abs_ret_5d", "DOW_Price_zscore_60d", "BAX_BankBoston_ret_20d__div__XLU_Util_ret_5d"], "is_new": false}, {"model_id": "v2_h7_STRESS_LightGBM_N14", "algo": "LightGBM", "regime": "STRESS", "horizon": 7, "n_features": 14, "F1_dir": 0.5651, "F1_UP_FORT": 0.1308, "F1_DOWN_FORT": 0.4899, "train_start": "2000-11-15", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["US3M_Rate_vol_20d", "AMD_zscore_60d__ret5x__XLU_Util_ret_5d", "DE_Deere_ret_5d", "VIX_Price_zscore_60d__macross__AMD_zscore_60d", "VIX_Price_zscore_60d__prod__JNJ_ret_20d", "JNJ_ret_20d__macross__vix_vs_ma5", "US30Y_Rate_ret_20d", "VRP_ma5", "DHR_vol_20d", "VIX_Price_zscore_60d__ret5x__JNJ_ret_20d", "vix_mean_abs_ret_5d", "DOW_Price_zscore_60d", "BAX_BankBoston_ret_20d__div__XLU_Util_ret_5d", "WMT_zscore_60d__div__BBY_BestBuy_zscore_60d"], "is_new": false}, {"model_id": "v2_h7_STRESS_LightGBM_N15", "algo": "LightGBM", "regime": "STRESS", "horizon": 7, "n_features": 15, "F1_dir": 0.52, "F1_UP_FORT": 0.0561, "F1_DOWN_FORT": 0.4467, "train_start": "2000-11-15", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["US3M_Rate_vol_20d", "AMD_zscore_60d__ret5x__XLU_Util_ret_5d", "DE_Deere_ret_5d", "VIX_Price_zscore_60d__macross__AMD_zscore_60d", "VIX_Price_zscore_60d__prod__JNJ_ret_20d", "JNJ_ret_20d__macross__vix_vs_ma5", "US30Y_Rate_ret_20d", "VRP_ma5", "DHR_vol_20d", "VIX_Price_zscore_60d__ret5x__JNJ_ret_20d", "vix_mean_abs_ret_5d", "DOW_Price_zscore_60d", "BAX_BankBoston_ret_20d__div__XLU_Util_ret_5d", "WMT_zscore_60d__div__BBY_BestBuy_zscore_60d", "FedFunds_zscore_60d"], "is_new": false}, {"model_id": "v2_h7_STRESS_LightGBM_N19", "algo": "LightGBM", "regime": "STRESS", "horizon": 7, "n_features": 19, "F1_dir": 0.5004, "F1_UP_FORT": 0.144, "F1_DOWN_FORT": 0.4422, "train_start": "2000-11-15", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["US3M_Rate_vol_20d", "AMD_zscore_60d__ret5x__XLU_Util_ret_5d", "DE_Deere_ret_5d", "VIX_Price_zscore_60d__macross__AMD_zscore_60d", "VIX_Price_zscore_60d__prod__JNJ_ret_20d", "JNJ_ret_20d__macross__vix_vs_ma5", "US30Y_Rate_ret_20d", "VRP_ma5", "DHR_vol_20d", "VIX_Price_zscore_60d__ret5x__JNJ_ret_20d", "vix_mean_abs_ret_5d", "DOW_Price_zscore_60d", "BAX_BankBoston_ret_20d__div__XLU_Util_ret_5d", "WMT_zscore_60d__div__BBY_BestBuy_zscore_60d", "FedFunds_zscore_60d", "US6M_Rate_ret_20d", "EFFR_ret_1d", "MSTR_Bitcoin3_ret_5d", "T10Y2Y_Spread_ret_5d__zrel__vix_vs_ma5"], "is_new": false}, {"model_id": "v2_h7_STRESS_GradientBoosting_N6", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 7, "n_features": 6, "F1_dir": 0.5003, "F1_UP_FORT": 0.2637, "F1_DOWN_FORT": 0.375, "train_start": "2000-11-15", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["US3M_Rate_vol_20d", "AMD_zscore_60d__ret5x__XLU_Util_ret_5d", "DE_Deere_ret_5d", "VIX_Price_zscore_60d__macross__AMD_zscore_60d", "VIX_Price_zscore_60d__prod__JNJ_ret_20d", "JNJ_ret_20d__macross__vix_vs_ma5"], "is_new": false}, {"model_id": "v2_h7_STRESS_GradientBoosting_N7", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 7, "n_features": 7, "F1_dir": 0.5666, "F1_UP_FORT": 0.3472, "F1_DOWN_FORT": 0.4014, "train_start": "2000-11-15", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["US3M_Rate_vol_20d", "AMD_zscore_60d__ret5x__XLU_Util_ret_5d", "DE_Deere_ret_5d", "VIX_Price_zscore_60d__macross__AMD_zscore_60d", "VIX_Price_zscore_60d__prod__JNJ_ret_20d", "JNJ_ret_20d__macross__vix_vs_ma5", "US30Y_Rate_ret_20d"], "is_new": false}, {"model_id": "v2_h7_STRESS_GradientBoosting_N8", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 7, "n_features": 8, "F1_dir": 0.5855, "F1_UP_FORT": 0.3239, "F1_DOWN_FORT": 0.3876, "train_start": "2000-11-15", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["US3M_Rate_vol_20d", "AMD_zscore_60d__ret5x__XLU_Util_ret_5d", "DE_Deere_ret_5d", "VIX_Price_zscore_60d__macross__AMD_zscore_60d", "VIX_Price_zscore_60d__prod__JNJ_ret_20d", "JNJ_ret_20d__macross__vix_vs_ma5", "US30Y_Rate_ret_20d", "VRP_ma5"], "is_new": false}, {"model_id": "v2_h7_STRESS_GradientBoosting_N9", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 7, "n_features": 9, "F1_dir": 0.5756, "F1_UP_FORT": 0.2205, "F1_DOWN_FORT": 0.4242, "train_start": "2000-11-15", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["US3M_Rate_vol_20d", "AMD_zscore_60d__ret5x__XLU_Util_ret_5d", "DE_Deere_ret_5d", "VIX_Price_zscore_60d__macross__AMD_zscore_60d", "VIX_Price_zscore_60d__prod__JNJ_ret_20d", "JNJ_ret_20d__macross__vix_vs_ma5", "US30Y_Rate_ret_20d", "VRP_ma5", "DHR_vol_20d"], "is_new": false}, {"model_id": "v2_h7_STRESS_GradientBoosting_N10", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 7, "n_features": 10, "F1_dir": 0.6001, "F1_UP_FORT": 0.2167, "F1_DOWN_FORT": 0.3895, "train_start": "2000-11-15", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["US3M_Rate_vol_20d", "AMD_zscore_60d__ret5x__XLU_Util_ret_5d", "DE_Deere_ret_5d", "VIX_Price_zscore_60d__macross__AMD_zscore_60d", "VIX_Price_zscore_60d__prod__JNJ_ret_20d", "JNJ_ret_20d__macross__vix_vs_ma5", "US30Y_Rate_ret_20d", "VRP_ma5", "DHR_vol_20d", "VIX_Price_zscore_60d__ret5x__JNJ_ret_20d"], "is_new": false}, {"model_id": "v2_h7_STRESS_GradientBoosting_N11", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 7, "n_features": 11, "F1_dir": 0.5408, "F1_UP_FORT": 0.1739, "F1_DOWN_FORT": 0.4354, "train_start": "2000-11-15", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["US3M_Rate_vol_20d", "AMD_zscore_60d__ret5x__XLU_Util_ret_5d", "DE_Deere_ret_5d", "VIX_Price_zscore_60d__macross__AMD_zscore_60d", "VIX_Price_zscore_60d__prod__JNJ_ret_20d", "JNJ_ret_20d__macross__vix_vs_ma5", "US30Y_Rate_ret_20d", "VRP_ma5", "DHR_vol_20d", "VIX_Price_zscore_60d__ret5x__JNJ_ret_20d", "vix_mean_abs_ret_5d"], "is_new": false}, {"model_id": "v2_h7_STRESS_GradientBoosting_N12", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 7, "n_features": 12, "F1_dir": 0.5193, "F1_UP_FORT": 0.1391, "F1_DOWN_FORT": 0.4514, "train_start": "2000-11-15", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["US3M_Rate_vol_20d", "AMD_zscore_60d__ret5x__XLU_Util_ret_5d", "DE_Deere_ret_5d", "VIX_Price_zscore_60d__macross__AMD_zscore_60d", "VIX_Price_zscore_60d__prod__JNJ_ret_20d", "JNJ_ret_20d__macross__vix_vs_ma5", "US30Y_Rate_ret_20d", "VRP_ma5", "DHR_vol_20d", "VIX_Price_zscore_60d__ret5x__JNJ_ret_20d", "vix_mean_abs_ret_5d", "DOW_Price_zscore_60d"], "is_new": false}, {"model_id": "v2_h7_STRESS_GradientBoosting_N13", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 7, "n_features": 13, "F1_dir": 0.5757, "F1_UP_FORT": 0.25, "F1_DOWN_FORT": 0.4604, "train_start": "2000-11-15", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["US3M_Rate_vol_20d", "AMD_zscore_60d__ret5x__XLU_Util_ret_5d", "DE_Deere_ret_5d", "VIX_Price_zscore_60d__macross__AMD_zscore_60d", "VIX_Price_zscore_60d__prod__JNJ_ret_20d", "JNJ_ret_20d__macross__vix_vs_ma5", "US30Y_Rate_ret_20d", "VRP_ma5", "DHR_vol_20d", "VIX_Price_zscore_60d__ret5x__JNJ_ret_20d", "vix_mean_abs_ret_5d", "DOW_Price_zscore_60d", "BAX_BankBoston_ret_20d__div__XLU_Util_ret_5d"], "is_new": false}, {"model_id": "v2_h7_STRESS_GradientBoosting_N14", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 7, "n_features": 14, "F1_dir": 0.5428, "F1_UP_FORT": 0.1579, "F1_DOWN_FORT": 0.4718, "train_start": "2000-11-15", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["US3M_Rate_vol_20d", "AMD_zscore_60d__ret5x__XLU_Util_ret_5d", "DE_Deere_ret_5d", "VIX_Price_zscore_60d__macross__AMD_zscore_60d", "VIX_Price_zscore_60d__prod__JNJ_ret_20d", "JNJ_ret_20d__macross__vix_vs_ma5", "US30Y_Rate_ret_20d", "VRP_ma5", "DHR_vol_20d", "VIX_Price_zscore_60d__ret5x__JNJ_ret_20d", "vix_mean_abs_ret_5d", "DOW_Price_zscore_60d", "BAX_BankBoston_ret_20d__div__XLU_Util_ret_5d", "WMT_zscore_60d__div__BBY_BestBuy_zscore_60d"], "is_new": false}, {"model_id": "v2_h7_STRESS_RandomForest_N5", "algo": "RandomForest", "regime": "STRESS", "horizon": 7, "n_features": 5, "F1_dir": 0.5023, "F1_UP_FORT": 0.2516, "F1_DOWN_FORT": 0.3485, "train_start": "2000-11-15", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["US3M_Rate_vol_20d", "AMD_zscore_60d__ret5x__XLU_Util_ret_5d", "DE_Deere_ret_5d", "VIX_Price_zscore_60d__macross__AMD_zscore_60d", "VIX_Price_zscore_60d__prod__JNJ_ret_20d"], "is_new": false}, {"model_id": "v2_h7_STRESS_RandomForest_N6", "algo": "RandomForest", "regime": "STRESS", "horizon": 7, "n_features": 6, "F1_dir": 0.5066, "F1_UP_FORT": 0.2312, "F1_DOWN_FORT": 0.3454, "train_start": "2000-11-15", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["US3M_Rate_vol_20d", "AMD_zscore_60d__ret5x__XLU_Util_ret_5d", "DE_Deere_ret_5d", "VIX_Price_zscore_60d__macross__AMD_zscore_60d", "VIX_Price_zscore_60d__prod__JNJ_ret_20d", "JNJ_ret_20d__macross__vix_vs_ma5"], "is_new": false}, {"model_id": "v2_h7_STRESS_RandomForest_N7", "algo": "RandomForest", "regime": "STRESS", "horizon": 7, "n_features": 7, "F1_dir": 0.529, "F1_UP_FORT": 0.2649, "F1_DOWN_FORT": 0.3456, "train_start": "2000-11-15", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["US3M_Rate_vol_20d", "AMD_zscore_60d__ret5x__XLU_Util_ret_5d", "DE_Deere_ret_5d", "VIX_Price_zscore_60d__macross__AMD_zscore_60d", "VIX_Price_zscore_60d__prod__JNJ_ret_20d", "JNJ_ret_20d__macross__vix_vs_ma5", "US30Y_Rate_ret_20d"], "is_new": false}, {"model_id": "v2_h7_STRESS_RandomForest_N8", "algo": "RandomForest", "regime": "STRESS", "horizon": 7, "n_features": 8, "F1_dir": 0.5493, "F1_UP_FORT": 0.2963, "F1_DOWN_FORT": 0.3609, "train_start": "2000-11-15", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["US3M_Rate_vol_20d", "AMD_zscore_60d__ret5x__XLU_Util_ret_5d", "DE_Deere_ret_5d", "VIX_Price_zscore_60d__macross__AMD_zscore_60d", "VIX_Price_zscore_60d__prod__JNJ_ret_20d", "JNJ_ret_20d__macross__vix_vs_ma5", "US30Y_Rate_ret_20d", "VRP_ma5"], "is_new": false}, {"model_id": "v2_h7_STRESS_RandomForest_N10", "algo": "RandomForest", "regime": "STRESS", "horizon": 7, "n_features": 10, "F1_dir": 0.5042, "F1_UP_FORT": 0.1185, "F1_DOWN_FORT": 0.3908, "train_start": "2000-11-15", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["US3M_Rate_vol_20d", "AMD_zscore_60d__ret5x__XLU_Util_ret_5d", "DE_Deere_ret_5d", "VIX_Price_zscore_60d__macross__AMD_zscore_60d", "VIX_Price_zscore_60d__prod__JNJ_ret_20d", "JNJ_ret_20d__macross__vix_vs_ma5", "US30Y_Rate_ret_20d", "VRP_ma5", "DHR_vol_20d", "VIX_Price_zscore_60d__ret5x__JNJ_ret_20d"], "is_new": false}, {"model_id": "v2_h7_STRESS_RandomForest_N11", "algo": "RandomForest", "regime": "STRESS", "horizon": 7, "n_features": 11, "F1_dir": 0.5112, "F1_UP_FORT": 0.1085, "F1_DOWN_FORT": 0.4878, "train_start": "2000-11-15", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["US3M_Rate_vol_20d", "AMD_zscore_60d__ret5x__XLU_Util_ret_5d", "DE_Deere_ret_5d", "VIX_Price_zscore_60d__macross__AMD_zscore_60d", "VIX_Price_zscore_60d__prod__JNJ_ret_20d", "JNJ_ret_20d__macross__vix_vs_ma5", "US30Y_Rate_ret_20d", "VRP_ma5", "DHR_vol_20d", "VIX_Price_zscore_60d__ret5x__JNJ_ret_20d", "vix_mean_abs_ret_5d"], "is_new": false}, {"model_id": "v2_h7_STRESS_LogisticRegression_N6", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 7, "n_features": 6, "F1_dir": 0.5146, "F1_UP_FORT": 0.0455, "F1_DOWN_FORT": 0.344, "train_start": "2000-11-15", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["US3M_Rate_vol_20d", "AMD_zscore_60d__ret5x__XLU_Util_ret_5d", "DE_Deere_ret_5d", "VIX_Price_zscore_60d__macross__AMD_zscore_60d", "VIX_Price_zscore_60d__prod__JNJ_ret_20d", "JNJ_ret_20d__macross__vix_vs_ma5"], "is_new": false}, {"model_id": "v2_h7_STRESS_LogisticRegression_N16", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 7, "n_features": 16, "F1_dir": 0.509, "F1_UP_FORT": 0.2411, "F1_DOWN_FORT": 0.413, "train_start": "2000-11-15", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["US3M_Rate_vol_20d", "AMD_zscore_60d__ret5x__XLU_Util_ret_5d", "DE_Deere_ret_5d", "VIX_Price_zscore_60d__macross__AMD_zscore_60d", "VIX_Price_zscore_60d__prod__JNJ_ret_20d", "JNJ_ret_20d__macross__vix_vs_ma5", "US30Y_Rate_ret_20d", "VRP_ma5", "DHR_vol_20d", "VIX_Price_zscore_60d__ret5x__JNJ_ret_20d", "vix_mean_abs_ret_5d", "DOW_Price_zscore_60d", "BAX_BankBoston_ret_20d__div__XLU_Util_ret_5d", "WMT_zscore_60d__div__BBY_BestBuy_zscore_60d", "FedFunds_zscore_60d", "US6M_Rate_ret_20d"], "is_new": false}, {"model_id": "v2_h7_STRESS_LogisticRegression_N17", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 7, "n_features": 17, "F1_dir": 0.5047, "F1_UP_FORT": 0.2286, "F1_DOWN_FORT": 0.4194, "train_start": "2000-11-15", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["US3M_Rate_vol_20d", "AMD_zscore_60d__ret5x__XLU_Util_ret_5d", "DE_Deere_ret_5d", "VIX_Price_zscore_60d__macross__AMD_zscore_60d", "VIX_Price_zscore_60d__prod__JNJ_ret_20d", "JNJ_ret_20d__macross__vix_vs_ma5", "US30Y_Rate_ret_20d", "VRP_ma5", "DHR_vol_20d", "VIX_Price_zscore_60d__ret5x__JNJ_ret_20d", "vix_mean_abs_ret_5d", "DOW_Price_zscore_60d", "BAX_BankBoston_ret_20d__div__XLU_Util_ret_5d", "WMT_zscore_60d__div__BBY_BestBuy_zscore_60d", "FedFunds_zscore_60d", "US6M_Rate_ret_20d", "EFFR_ret_1d"], "is_new": false}, {"model_id": "v2_h7_STRESS_LogisticRegression_N18", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 7, "n_features": 18, "F1_dir": 0.5162, "F1_UP_FORT": 0.2128, "F1_DOWN_FORT": 0.4427, "train_start": "2000-11-15", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["US3M_Rate_vol_20d", "AMD_zscore_60d__ret5x__XLU_Util_ret_5d", "DE_Deere_ret_5d", "VIX_Price_zscore_60d__macross__AMD_zscore_60d", "VIX_Price_zscore_60d__prod__JNJ_ret_20d", "JNJ_ret_20d__macross__vix_vs_ma5", "US30Y_Rate_ret_20d", "VRP_ma5", "DHR_vol_20d", "VIX_Price_zscore_60d__ret5x__JNJ_ret_20d", "vix_mean_abs_ret_5d", "DOW_Price_zscore_60d", "BAX_BankBoston_ret_20d__div__XLU_Util_ret_5d", "WMT_zscore_60d__div__BBY_BestBuy_zscore_60d", "FedFunds_zscore_60d", "US6M_Rate_ret_20d", "EFFR_ret_1d", "MSTR_Bitcoin3_ret_5d"], "is_new": false}, {"model_id": "v2_h7_STRESS_LogisticRegression_N19", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 7, "n_features": 19, "F1_dir": 0.5401, "F1_UP_FORT": 0.2615, "F1_DOWN_FORT": 0.4494, "train_start": "2000-11-15", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["US3M_Rate_vol_20d", "AMD_zscore_60d__ret5x__XLU_Util_ret_5d", "DE_Deere_ret_5d", "VIX_Price_zscore_60d__macross__AMD_zscore_60d", "VIX_Price_zscore_60d__prod__JNJ_ret_20d", "JNJ_ret_20d__macross__vix_vs_ma5", "US30Y_Rate_ret_20d", "VRP_ma5", "DHR_vol_20d", "VIX_Price_zscore_60d__ret5x__JNJ_ret_20d", "vix_mean_abs_ret_5d", "DOW_Price_zscore_60d", "BAX_BankBoston_ret_20d__div__XLU_Util_ret_5d", "WMT_zscore_60d__div__BBY_BestBuy_zscore_60d", "FedFunds_zscore_60d", "US6M_Rate_ret_20d", "EFFR_ret_1d", "MSTR_Bitcoin3_ret_5d", "T10Y2Y_Spread_ret_5d__zrel__vix_vs_ma5"], "is_new": false}, {"model_id": "v2_h7_STRESS_LogisticRegression_N20", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 7, "n_features": 20, "F1_dir": 0.5098, "F1_UP_FORT": 0.2424, "F1_DOWN_FORT": 0.4335, "train_start": "2000-11-15", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["US3M_Rate_vol_20d", "AMD_zscore_60d__ret5x__XLU_Util_ret_5d", "DE_Deere_ret_5d", "VIX_Price_zscore_60d__macross__AMD_zscore_60d", "VIX_Price_zscore_60d__prod__JNJ_ret_20d", "JNJ_ret_20d__macross__vix_vs_ma5", "US30Y_Rate_ret_20d", "VRP_ma5", "DHR_vol_20d", "VIX_Price_zscore_60d__ret5x__JNJ_ret_20d", "vix_mean_abs_ret_5d", "DOW_Price_zscore_60d", "BAX_BankBoston_ret_20d__div__XLU_Util_ret_5d", "WMT_zscore_60d__div__BBY_BestBuy_zscore_60d", "FedFunds_zscore_60d", "US6M_Rate_ret_20d", "EFFR_ret_1d", "MSTR_Bitcoin3_ret_5d", "T10Y2Y_Spread_ret_5d__zrel__vix_vs_ma5", "WMT_zscore_60d__div__spx_momentum_3d"], "is_new": false}, {"model_id": "v2_h7_STRESS_XGBoost_Optuna_N8", "algo": "XGBoost", "regime": "STRESS", "horizon": 7, "n_features": 8, "F1_dir": 0.5479, "F1_UP_FORT": 0.3057, "F1_DOWN_FORT": 0.4408, "train_start": "2000-11-15", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["US3M_Rate_vol_20d", "AMD_zscore_60d__ret5x__XLU_Util_ret_5d", "DE_Deere_ret_5d", "VIX_Price_zscore_60d__macross__AMD_zscore_60d", "VIX_Price_zscore_60d__prod__JNJ_ret_20d", "JNJ_ret_20d__macross__vix_vs_ma5", "US30Y_Rate_ret_20d", "VRP_ma5"], "is_new": false}, {"model_id": "v2_h7_STRESS_XGBoost_OptunaCal_N8", "algo": "XGBoostCal", "regime": "STRESS", "horizon": 7, "n_features": 8, "F1_dir": 0.5661, "F1_UP_FORT": 0.2946, "F1_DOWN_FORT": 0.371, "train_start": "2000-11-15", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["US3M_Rate_vol_20d", "AMD_zscore_60d__ret5x__XLU_Util_ret_5d", "DE_Deere_ret_5d", "VIX_Price_zscore_60d__macross__AMD_zscore_60d", "VIX_Price_zscore_60d__prod__JNJ_ret_20d", "JNJ_ret_20d__macross__vix_vs_ma5", "US30Y_Rate_ret_20d", "VRP_ma5"], "is_new": false}, {"model_id": "v2_h7_GLOBAL_XGBoost_N5", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 7, "n_features": 5, "F1_dir": 0.5494, "F1_UP_FORT": 0.2545, "F1_DOWN_FORT": 0.4396, "train_start": "2001-02-06", "sampler": "SMOTE", "best_params": "{}", "features": ["VIX_Price_zscore_60d__div__NFCI_vol_20d", "Core_CPI_zscore_60d", "VIX_Price_zscore_60d__div__CTAS_Cintas_vol_20d", "Nikkei_Japan_zscore_60d", "XLK_Tech_zscore_60d"], "is_new": false}, {"model_id": "v2_h7_GLOBAL_XGBoost_N6", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 7, "n_features": 6, "F1_dir": 0.5514, "F1_UP_FORT": 0.2583, "F1_DOWN_FORT": 0.4245, "train_start": "2001-02-06", "sampler": "SMOTE", "best_params": "{}", "features": ["VIX_Price_zscore_60d__div__NFCI_vol_20d", "Core_CPI_zscore_60d", "VIX_Price_zscore_60d__div__CTAS_Cintas_vol_20d", "Nikkei_Japan_zscore_60d", "XLK_Tech_zscore_60d", "NOC_Northrop_ret_20d"], "is_new": false}, {"model_id": "v2_h7_GLOBAL_XGBoost_N7", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 7, "n_features": 7, "F1_dir": 0.518, "F1_UP_FORT": 0.2507, "F1_DOWN_FORT": 0.3848, "train_start": "2001-02-06", "sampler": "SMOTE", "best_params": "{}", "features": ["VIX_Price_zscore_60d__div__NFCI_vol_20d", "Core_CPI_zscore_60d", "VIX_Price_zscore_60d__div__CTAS_Cintas_vol_20d", "Nikkei_Japan_zscore_60d", "XLK_Tech_zscore_60d", "NOC_Northrop_ret_20d", "CLX_Clorox_vol_20d"], "is_new": false}, {"model_id": "v2_h7_GLOBAL_XGBoost_N8", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 7, "n_features": 8, "F1_dir": 0.529, "F1_UP_FORT": 0.2266, "F1_DOWN_FORT": 0.4054, "train_start": "2001-02-06", "sampler": "SMOTE", "best_params": "{}", "features": ["VIX_Price_zscore_60d__div__NFCI_vol_20d", "Core_CPI_zscore_60d", "VIX_Price_zscore_60d__div__CTAS_Cintas_vol_20d", "Nikkei_Japan_zscore_60d", "XLK_Tech_zscore_60d", "NOC_Northrop_ret_20d", "CLX_Clorox_vol_20d", "vix_zscore_10d__div__heston_var_ev_h7"], "is_new": false}, {"model_id": "v2_h7_GLOBAL_XGBoost_N9", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 7, "n_features": 9, "F1_dir": 0.5553, "F1_UP_FORT": 0.3178, "F1_DOWN_FORT": 0.4278, "train_start": "2001-02-06", "sampler": "SMOTE", "best_params": "{}", "features": ["VIX_Price_zscore_60d__div__NFCI_vol_20d", "Core_CPI_zscore_60d", "VIX_Price_zscore_60d__div__CTAS_Cintas_vol_20d", "Nikkei_Japan_zscore_60d", "XLK_Tech_zscore_60d", "NOC_Northrop_ret_20d", "CLX_Clorox_vol_20d", "vix_zscore_10d__div__heston_var_ev_h7", "Nikkei_Japan_vol_20d"], "is_new": false}, {"model_id": "v2_h7_GLOBAL_XGBoost_N10", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 7, "n_features": 10, "F1_dir": 0.5347, "F1_UP_FORT": 0.3087, "F1_DOWN_FORT": 0.4403, "train_start": "2001-02-06", "sampler": "SMOTE", "best_params": "{}", "features": ["VIX_Price_zscore_60d__div__NFCI_vol_20d", "Core_CPI_zscore_60d", "VIX_Price_zscore_60d__div__CTAS_Cintas_vol_20d", "Nikkei_Japan_zscore_60d", "XLK_Tech_zscore_60d", "NOC_Northrop_ret_20d", "CLX_Clorox_vol_20d", "vix_zscore_10d__div__heston_var_ev_h7", "Nikkei_Japan_vol_20d", "vix_vol_of_vol_10d__minus__NFCI_vol_20d"], "is_new": false}, {"model_id": "v2_h7_GLOBAL_XGBoost_N11", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 7, "n_features": 11, "F1_dir": 0.5482, "F1_UP_FORT": 0.2893, "F1_DOWN_FORT": 0.4424, "train_start": "2001-02-06", "sampler": "SMOTE", "best_params": "{}", "features": ["VIX_Price_zscore_60d__div__NFCI_vol_20d", "Core_CPI_zscore_60d", "VIX_Price_zscore_60d__div__CTAS_Cintas_vol_20d", "Nikkei_Japan_zscore_60d", "XLK_Tech_zscore_60d", "NOC_Northrop_ret_20d", "CLX_Clorox_vol_20d", "vix_zscore_10d__div__heston_var_ev_h7", "Nikkei_Japan_vol_20d", "vix_vol_of_vol_10d__minus__NFCI_vol_20d", "VRP__macross__hmm_p_stress"], "is_new": false}, {"model_id": "v2_h7_GLOBAL_XGBoost_N12", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 7, "n_features": 12, "F1_dir": 0.5453, "F1_UP_FORT": 0.3021, "F1_DOWN_FORT": 0.4322, "train_start": "2001-02-06", "sampler": "SMOTE", "best_params": "{}", "features": ["VIX_Price_zscore_60d__div__NFCI_vol_20d", "Core_CPI_zscore_60d", "VIX_Price_zscore_60d__div__CTAS_Cintas_vol_20d", "Nikkei_Japan_zscore_60d", "XLK_Tech_zscore_60d", "NOC_Northrop_ret_20d", "CLX_Clorox_vol_20d", "vix_zscore_10d__div__heston_var_ev_h7", "Nikkei_Japan_vol_20d", "vix_vol_of_vol_10d__minus__NFCI_vol_20d", "VRP__macross__hmm_p_stress", "HD_zscore_60d"], "is_new": false}, {"model_id": "v2_h7_GLOBAL_XGBoost_N13", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 7, "n_features": 13, "F1_dir": 0.5301, "F1_UP_FORT": 0.2872, "F1_DOWN_FORT": 0.4388, "train_start": "2001-02-06", "sampler": "SMOTE", "best_params": "{}", "features": ["VIX_Price_zscore_60d__div__NFCI_vol_20d", "Core_CPI_zscore_60d", "VIX_Price_zscore_60d__div__CTAS_Cintas_vol_20d", "Nikkei_Japan_zscore_60d", "XLK_Tech_zscore_60d", "NOC_Northrop_ret_20d", "CLX_Clorox_vol_20d", "vix_zscore_10d__div__heston_var_ev_h7", "Nikkei_Japan_vol_20d", "vix_vol_of_vol_10d__minus__NFCI_vol_20d", "VRP__macross__hmm_p_stress", "HD_zscore_60d", "Industrial_Production_zscore_60d"], "is_new": false}, {"model_id": "v2_h7_GLOBAL_XGBoost_N14", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 7, "n_features": 14, "F1_dir": 0.5998, "F1_UP_FORT": 0.3621, "F1_DOWN_FORT": 0.4649, "train_start": "2001-02-06", "sampler": "SMOTE", "best_params": "{}", "features": ["VIX_Price_zscore_60d__div__NFCI_vol_20d", "Core_CPI_zscore_60d", "VIX_Price_zscore_60d__div__CTAS_Cintas_vol_20d", "Nikkei_Japan_zscore_60d", "XLK_Tech_zscore_60d", "NOC_Northrop_ret_20d", "CLX_Clorox_vol_20d", "vix_zscore_10d__div__heston_var_ev_h7", "Nikkei_Japan_vol_20d", "vix_vol_of_vol_10d__minus__NFCI_vol_20d", "VRP__macross__hmm_p_stress", "HD_zscore_60d", "Industrial_Production_zscore_60d", "NFCI_ret_5d__div__NFCI_vol_20d"], "is_new": false}, {"model_id": "v2_h7_GLOBAL_XGBoost_N15", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 7, "n_features": 15, "F1_dir": 0.5945, "F1_UP_FORT": 0.3436, "F1_DOWN_FORT": 0.4506, "train_start": "2001-02-06", "sampler": "SMOTE", "best_params": "{}", "features": ["VIX_Price_zscore_60d__div__NFCI_vol_20d", "Core_CPI_zscore_60d", "VIX_Price_zscore_60d__div__CTAS_Cintas_vol_20d", "Nikkei_Japan_zscore_60d", "XLK_Tech_zscore_60d", "NOC_Northrop_ret_20d", "CLX_Clorox_vol_20d", "vix_zscore_10d__div__heston_var_ev_h7", "Nikkei_Japan_vol_20d", "vix_vol_of_vol_10d__minus__NFCI_vol_20d", "VRP__macross__hmm_p_stress", "HD_zscore_60d", "Industrial_Production_zscore_60d", "NFCI_ret_5d__div__NFCI_vol_20d", "ASX_Australia_vol_20d"], "is_new": false}, {"model_id": "v2_h7_GLOBAL_XGBoost_N16", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 7, "n_features": 16, "F1_dir": 0.6114, "F1_UP_FORT": 0.3699, "F1_DOWN_FORT": 0.4701, "train_start": "2001-02-06", "sampler": "SMOTE", "best_params": "{}", "features": ["VIX_Price_zscore_60d__div__NFCI_vol_20d", "Core_CPI_zscore_60d", "VIX_Price_zscore_60d__div__CTAS_Cintas_vol_20d", "Nikkei_Japan_zscore_60d", "XLK_Tech_zscore_60d", "NOC_Northrop_ret_20d", "CLX_Clorox_vol_20d", "vix_zscore_10d__div__heston_var_ev_h7", "Nikkei_Japan_vol_20d", "vix_vol_of_vol_10d__minus__NFCI_vol_20d", "VRP__macross__hmm_p_stress", "HD_zscore_60d", "Industrial_Production_zscore_60d", "NFCI_ret_5d__div__NFCI_vol_20d", "ASX_Australia_vol_20d", "Michigan_Sentiment_ret_20d"], "is_new": false}, {"model_id": "v2_h7_GLOBAL_XGBoost_N17", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 7, "n_features": 17, "F1_dir": 0.606, "F1_UP_FORT": 0.3881, "F1_DOWN_FORT": 0.4709, "train_start": "2001-02-06", "sampler": "SMOTE", "best_params": "{}", "features": ["VIX_Price_zscore_60d__div__NFCI_vol_20d", "Core_CPI_zscore_60d", "VIX_Price_zscore_60d__div__CTAS_Cintas_vol_20d", "Nikkei_Japan_zscore_60d", "XLK_Tech_zscore_60d", "NOC_Northrop_ret_20d", "CLX_Clorox_vol_20d", "vix_zscore_10d__div__heston_var_ev_h7", "Nikkei_Japan_vol_20d", "vix_vol_of_vol_10d__minus__NFCI_vol_20d", "VRP__macross__hmm_p_stress", "HD_zscore_60d", "Industrial_Production_zscore_60d", "NFCI_ret_5d__div__NFCI_vol_20d", "ASX_Australia_vol_20d", "Michigan_Sentiment_ret_20d", "XOM_ret_20d"], "is_new": false}, {"model_id": "v2_h7_GLOBAL_XGBoost_N18", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 7, "n_features": 18, "F1_dir": 0.6114, "F1_UP_FORT": 0.3894, "F1_DOWN_FORT": 0.4895, "train_start": "2001-02-06", "sampler": "SMOTE", "best_params": "{}", "features": ["VIX_Price_zscore_60d__div__NFCI_vol_20d", "Core_CPI_zscore_60d", "VIX_Price_zscore_60d__div__CTAS_Cintas_vol_20d", "Nikkei_Japan_zscore_60d", "XLK_Tech_zscore_60d", "NOC_Northrop_ret_20d", "CLX_Clorox_vol_20d", "vix_zscore_10d__div__heston_var_ev_h7", "Nikkei_Japan_vol_20d", "vix_vol_of_vol_10d__minus__NFCI_vol_20d", "VRP__macross__hmm_p_stress", "HD_zscore_60d", "Industrial_Production_zscore_60d", "NFCI_ret_5d__div__NFCI_vol_20d", "ASX_Australia_vol_20d", "Michigan_Sentiment_ret_20d", "XOM_ret_20d", "VIX_Price_zscore_60d__ret5x__NFCI_zscore_60d"], "is_new": false}, {"model_id": "v2_h7_GLOBAL_XGBoost_N19", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 7, "n_features": 19, "F1_dir": 0.5985, "F1_UP_FORT": 0.3755, "F1_DOWN_FORT": 0.4777, "train_start": "2001-02-06", "sampler": "SMOTE", "best_params": "{}", "features": ["VIX_Price_zscore_60d__div__NFCI_vol_20d", "Core_CPI_zscore_60d", "VIX_Price_zscore_60d__div__CTAS_Cintas_vol_20d", "Nikkei_Japan_zscore_60d", "XLK_Tech_zscore_60d", "NOC_Northrop_ret_20d", "CLX_Clorox_vol_20d", "vix_zscore_10d__div__heston_var_ev_h7", "Nikkei_Japan_vol_20d", "vix_vol_of_vol_10d__minus__NFCI_vol_20d", "VRP__macross__hmm_p_stress", "HD_zscore_60d", "Industrial_Production_zscore_60d", "NFCI_ret_5d__div__NFCI_vol_20d", "ASX_Australia_vol_20d", "Michigan_Sentiment_ret_20d", "XOM_ret_20d", "VIX_Price_zscore_60d__ret5x__NFCI_zscore_60d", "VIX_Price_zscore_60d__div__hmm_p_stress"], "is_new": false}, {"model_id": "v2_h7_GLOBAL_XGBoost_N20", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 7, "n_features": 20, "F1_dir": 0.6197, "F1_UP_FORT": 0.4034, "F1_DOWN_FORT": 0.492, "train_start": "2001-02-06", "sampler": "SMOTE", "best_params": "{}", "features": ["VIX_Price_zscore_60d__div__NFCI_vol_20d", "Core_CPI_zscore_60d", "VIX_Price_zscore_60d__div__CTAS_Cintas_vol_20d", "Nikkei_Japan_zscore_60d", "XLK_Tech_zscore_60d", "NOC_Northrop_ret_20d", "CLX_Clorox_vol_20d", "vix_zscore_10d__div__heston_var_ev_h7", "Nikkei_Japan_vol_20d", "vix_vol_of_vol_10d__minus__NFCI_vol_20d", "VRP__macross__hmm_p_stress", "HD_zscore_60d", "Industrial_Production_zscore_60d", "NFCI_ret_5d__div__NFCI_vol_20d", "ASX_Australia_vol_20d", "Michigan_Sentiment_ret_20d", "XOM_ret_20d", "VIX_Price_zscore_60d__ret5x__NFCI_zscore_60d", "VIX_Price_zscore_60d__div__hmm_p_stress", "VIX_Price_zscore_60d__prod__CTAS_Cintas_vol_20d"], "is_new": false}, {"model_id": "v2_h7_GLOBAL_XGBoost_N21", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 7, "n_features": 21, "F1_dir": 0.6106, "F1_UP_FORT": 0.3764, "F1_DOWN_FORT": 0.4872, "train_start": "2001-02-06", "sampler": "SMOTE", "best_params": "{}", "features": ["VIX_Price_zscore_60d__div__NFCI_vol_20d", "Core_CPI_zscore_60d", "VIX_Price_zscore_60d__div__CTAS_Cintas_vol_20d", "Nikkei_Japan_zscore_60d", "XLK_Tech_zscore_60d", "NOC_Northrop_ret_20d", "CLX_Clorox_vol_20d", "vix_zscore_10d__div__heston_var_ev_h7", "Nikkei_Japan_vol_20d", "vix_vol_of_vol_10d__minus__NFCI_vol_20d", "VRP__macross__hmm_p_stress", "HD_zscore_60d", "Industrial_Production_zscore_60d", "NFCI_ret_5d__div__NFCI_vol_20d", "ASX_Australia_vol_20d", "Michigan_Sentiment_ret_20d", "XOM_ret_20d", "VIX_Price_zscore_60d__ret5x__NFCI_zscore_60d", "VIX_Price_zscore_60d__div__hmm_p_stress", "VIX_Price_zscore_60d__prod__CTAS_Cintas_vol_20d", "vix_zscore_10d__div__vix_vol_of_vol_10d"], "is_new": false}, {"model_id": "v2_h7_GLOBAL_XGBoost_N22", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 7, "n_features": 22, "F1_dir": 0.6106, "F1_UP_FORT": 0.3827, "F1_DOWN_FORT": 0.4832, "train_start": "2001-02-06", "sampler": "SMOTE", "best_params": "{}", "features": ["VIX_Price_zscore_60d__div__NFCI_vol_20d", "Core_CPI_zscore_60d", "VIX_Price_zscore_60d__div__CTAS_Cintas_vol_20d", "Nikkei_Japan_zscore_60d", "XLK_Tech_zscore_60d", "NOC_Northrop_ret_20d", "CLX_Clorox_vol_20d", "vix_zscore_10d__div__heston_var_ev_h7", "Nikkei_Japan_vol_20d", "vix_vol_of_vol_10d__minus__NFCI_vol_20d", "VRP__macross__hmm_p_stress", "HD_zscore_60d", "Industrial_Production_zscore_60d", "NFCI_ret_5d__div__NFCI_vol_20d", "ASX_Australia_vol_20d", "Michigan_Sentiment_ret_20d", "XOM_ret_20d", "VIX_Price_zscore_60d__ret5x__NFCI_zscore_60d", "VIX_Price_zscore_60d__div__hmm_p_stress", "VIX_Price_zscore_60d__prod__CTAS_Cintas_vol_20d", "vix_zscore_10d__div__vix_vol_of_vol_10d", "VIX_Price_zscore_60d__zrel__SBUX_ret_5d"], "is_new": false}, {"model_id": "v2_h7_GLOBAL_XGBoost_N23", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 7, "n_features": 23, "F1_dir": 0.5999, "F1_UP_FORT": 0.3645, "F1_DOWN_FORT": 0.4932, "train_start": "2001-02-06", "sampler": "SMOTE", "best_params": "{}", "features": ["VIX_Price_zscore_60d__div__NFCI_vol_20d", "Core_CPI_zscore_60d", "VIX_Price_zscore_60d__div__CTAS_Cintas_vol_20d", "Nikkei_Japan_zscore_60d", "XLK_Tech_zscore_60d", "NOC_Northrop_ret_20d", "CLX_Clorox_vol_20d", "vix_zscore_10d__div__heston_var_ev_h7", "Nikkei_Japan_vol_20d", "vix_vol_of_vol_10d__minus__NFCI_vol_20d", "VRP__macross__hmm_p_stress", "HD_zscore_60d", "Industrial_Production_zscore_60d", "NFCI_ret_5d__div__NFCI_vol_20d", "ASX_Australia_vol_20d", "Michigan_Sentiment_ret_20d", "XOM_ret_20d", "VIX_Price_zscore_60d__ret5x__NFCI_zscore_60d", "VIX_Price_zscore_60d__div__hmm_p_stress", "VIX_Price_zscore_60d__prod__CTAS_Cintas_vol_20d", "vix_zscore_10d__div__vix_vol_of_vol_10d", "VIX_Price_zscore_60d__zrel__SBUX_ret_5d", "VIX_Price_zscore_60d__minus__heston_xi"], "is_new": false}, {"model_id": "v2_h7_GLOBAL_LightGBM_N5", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 7, "n_features": 5, "F1_dir": 0.5485, "F1_UP_FORT": 0.261, "F1_DOWN_FORT": 0.4321, "train_start": "2001-02-06", "sampler": "SMOTE", "best_params": "{}", "features": ["VIX_Price_zscore_60d__div__NFCI_vol_20d", "Core_CPI_zscore_60d", "VIX_Price_zscore_60d__div__CTAS_Cintas_vol_20d", "Nikkei_Japan_zscore_60d", "XLK_Tech_zscore_60d"], "is_new": false}, {"model_id": "v2_h7_GLOBAL_LightGBM_N6", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 7, "n_features": 6, "F1_dir": 0.5528, "F1_UP_FORT": 0.2349, "F1_DOWN_FORT": 0.4146, "train_start": "2001-02-06", "sampler": "SMOTE", "best_params": "{}", "features": ["VIX_Price_zscore_60d__div__NFCI_vol_20d", "Core_CPI_zscore_60d", "VIX_Price_zscore_60d__div__CTAS_Cintas_vol_20d", "Nikkei_Japan_zscore_60d", "XLK_Tech_zscore_60d", "NOC_Northrop_ret_20d"], "is_new": false}, {"model_id": "v2_h7_GLOBAL_LightGBM_N7", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 7, "n_features": 7, "F1_dir": 0.5187, "F1_UP_FORT": 0.2047, "F1_DOWN_FORT": 0.3705, "train_start": "2001-02-06", "sampler": "SMOTE", "best_params": "{}", "features": ["VIX_Price_zscore_60d__div__NFCI_vol_20d", "Core_CPI_zscore_60d", "VIX_Price_zscore_60d__div__CTAS_Cintas_vol_20d", "Nikkei_Japan_zscore_60d", "XLK_Tech_zscore_60d", "NOC_Northrop_ret_20d", "CLX_Clorox_vol_20d"], "is_new": false}, {"model_id": "v2_h7_GLOBAL_LightGBM_N8", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 7, "n_features": 8, "F1_dir": 0.5295, "F1_UP_FORT": 0.1964, "F1_DOWN_FORT": 0.3897, "train_start": "2001-02-06", "sampler": "SMOTE", "best_params": "{}", "features": ["VIX_Price_zscore_60d__div__NFCI_vol_20d", "Core_CPI_zscore_60d", "VIX_Price_zscore_60d__div__CTAS_Cintas_vol_20d", "Nikkei_Japan_zscore_60d", "XLK_Tech_zscore_60d", "NOC_Northrop_ret_20d", "CLX_Clorox_vol_20d", "vix_zscore_10d__div__heston_var_ev_h7"], "is_new": false}, {"model_id": "v2_h7_GLOBAL_LightGBM_N9", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 7, "n_features": 9, "F1_dir": 0.5572, "F1_UP_FORT": 0.2967, "F1_DOWN_FORT": 0.4451, "train_start": "2001-02-06", "sampler": "SMOTE", "best_params": "{}", "features": ["VIX_Price_zscore_60d__div__NFCI_vol_20d", "Core_CPI_zscore_60d", "VIX_Price_zscore_60d__div__CTAS_Cintas_vol_20d", "Nikkei_Japan_zscore_60d", "XLK_Tech_zscore_60d", "NOC_Northrop_ret_20d", "CLX_Clorox_vol_20d", "vix_zscore_10d__div__heston_var_ev_h7", "Nikkei_Japan_vol_20d"], "is_new": false}, {"model_id": "v2_h7_GLOBAL_LightGBM_N10", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 7, "n_features": 10, "F1_dir": 0.5396, "F1_UP_FORT": 0.2915, "F1_DOWN_FORT": 0.4544, "train_start": "2001-02-06", "sampler": "SMOTE", "best_params": "{}", "features": ["VIX_Price_zscore_60d__div__NFCI_vol_20d", "Core_CPI_zscore_60d", "VIX_Price_zscore_60d__div__CTAS_Cintas_vol_20d", "Nikkei_Japan_zscore_60d", "XLK_Tech_zscore_60d", "NOC_Northrop_ret_20d", "CLX_Clorox_vol_20d", "vix_zscore_10d__div__heston_var_ev_h7", "Nikkei_Japan_vol_20d", "vix_vol_of_vol_10d__minus__NFCI_vol_20d"], "is_new": false}, {"model_id": "v2_h7_GLOBAL_LightGBM_N11", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 7, "n_features": 11, "F1_dir": 0.5168, "F1_UP_FORT": 0.2776, "F1_DOWN_FORT": 0.4076, "train_start": "2001-02-06", "sampler": "SMOTE", "best_params": "{}", "features": ["VIX_Price_zscore_60d__div__NFCI_vol_20d", "Core_CPI_zscore_60d", "VIX_Price_zscore_60d__div__CTAS_Cintas_vol_20d", "Nikkei_Japan_zscore_60d", "XLK_Tech_zscore_60d", "NOC_Northrop_ret_20d", "CLX_Clorox_vol_20d", "vix_zscore_10d__div__heston_var_ev_h7", "Nikkei_Japan_vol_20d", "vix_vol_of_vol_10d__minus__NFCI_vol_20d", "VRP__macross__hmm_p_stress"], "is_new": false}, {"model_id": "v2_h7_GLOBAL_LightGBM_N12", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 7, "n_features": 12, "F1_dir": 0.5427, "F1_UP_FORT": 0.295, "F1_DOWN_FORT": 0.4337, "train_start": "2001-02-06", "sampler": "SMOTE", "best_params": "{}", "features": ["VIX_Price_zscore_60d__div__NFCI_vol_20d", "Core_CPI_zscore_60d", "VIX_Price_zscore_60d__div__CTAS_Cintas_vol_20d", "Nikkei_Japan_zscore_60d", "XLK_Tech_zscore_60d", "NOC_Northrop_ret_20d", "CLX_Clorox_vol_20d", "vix_zscore_10d__div__heston_var_ev_h7", "Nikkei_Japan_vol_20d", "vix_vol_of_vol_10d__minus__NFCI_vol_20d", "VRP__macross__hmm_p_stress", "HD_zscore_60d"], "is_new": false}, {"model_id": "v2_h7_GLOBAL_LightGBM_N13", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 7, "n_features": 13, "F1_dir": 0.565, "F1_UP_FORT": 0.2968, "F1_DOWN_FORT": 0.4545, "train_start": "2001-02-06", "sampler": "SMOTE", "best_params": "{}", "features": ["VIX_Price_zscore_60d__div__NFCI_vol_20d", "Core_CPI_zscore_60d", "VIX_Price_zscore_60d__div__CTAS_Cintas_vol_20d", "Nikkei_Japan_zscore_60d", "XLK_Tech_zscore_60d", "NOC_Northrop_ret_20d", "CLX_Clorox_vol_20d", "vix_zscore_10d__div__heston_var_ev_h7", "Nikkei_Japan_vol_20d", "vix_vol_of_vol_10d__minus__NFCI_vol_20d", "VRP__macross__hmm_p_stress", "HD_zscore_60d", "Industrial_Production_zscore_60d"], "is_new": false}, {"model_id": "v2_h7_GLOBAL_LightGBM_N14", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 7, "n_features": 14, "F1_dir": 0.6013, "F1_UP_FORT": 0.3317, "F1_DOWN_FORT": 0.4679, "train_start": "2001-02-06", "sampler": "SMOTE", "best_params": "{}", "features": ["VIX_Price_zscore_60d__div__NFCI_vol_20d", "Core_CPI_zscore_60d", "VIX_Price_zscore_60d__div__CTAS_Cintas_vol_20d", "Nikkei_Japan_zscore_60d", "XLK_Tech_zscore_60d", "NOC_Northrop_ret_20d", "CLX_Clorox_vol_20d", "vix_zscore_10d__div__heston_var_ev_h7", "Nikkei_Japan_vol_20d", "vix_vol_of_vol_10d__minus__NFCI_vol_20d", "VRP__macross__hmm_p_stress", "HD_zscore_60d", "Industrial_Production_zscore_60d", "NFCI_ret_5d__div__NFCI_vol_20d"], "is_new": false}, {"model_id": "v2_h7_GLOBAL_LightGBM_N15", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 7, "n_features": 15, "F1_dir": 0.5815, "F1_UP_FORT": 0.3333, "F1_DOWN_FORT": 0.4605, "train_start": "2001-02-06", "sampler": "SMOTE", "best_params": "{}", "features": ["VIX_Price_zscore_60d__div__NFCI_vol_20d", "Core_CPI_zscore_60d", "VIX_Price_zscore_60d__div__CTAS_Cintas_vol_20d", "Nikkei_Japan_zscore_60d", "XLK_Tech_zscore_60d", "NOC_Northrop_ret_20d", "CLX_Clorox_vol_20d", "vix_zscore_10d__div__heston_var_ev_h7", "Nikkei_Japan_vol_20d", "vix_vol_of_vol_10d__minus__NFCI_vol_20d", "VRP__macross__hmm_p_stress", "HD_zscore_60d", "Industrial_Production_zscore_60d", "NFCI_ret_5d__div__NFCI_vol_20d", "ASX_Australia_vol_20d"], "is_new": false}, {"model_id": "v2_h7_GLOBAL_LightGBM_N16", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 7, "n_features": 16, "F1_dir": 0.604, "F1_UP_FORT": 0.3936, "F1_DOWN_FORT": 0.468, "train_start": "2001-02-06", "sampler": "SMOTE", "best_params": "{}", "features": ["VIX_Price_zscore_60d__div__NFCI_vol_20d", "Core_CPI_zscore_60d", "VIX_Price_zscore_60d__div__CTAS_Cintas_vol_20d", "Nikkei_Japan_zscore_60d", "XLK_Tech_zscore_60d", "NOC_Northrop_ret_20d", "CLX_Clorox_vol_20d", "vix_zscore_10d__div__heston_var_ev_h7", "Nikkei_Japan_vol_20d", "vix_vol_of_vol_10d__minus__NFCI_vol_20d", "VRP__macross__hmm_p_stress", "HD_zscore_60d", "Industrial_Production_zscore_60d", "NFCI_ret_5d__div__NFCI_vol_20d", "ASX_Australia_vol_20d", "Michigan_Sentiment_ret_20d"], "is_new": false}, {"model_id": "v2_h7_GLOBAL_LightGBM_N17", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 7, "n_features": 17, "F1_dir": 0.592, "F1_UP_FORT": 0.3816, "F1_DOWN_FORT": 0.4631, "train_start": "2001-02-06", "sampler": "SMOTE", "best_params": "{}", "features": ["VIX_Price_zscore_60d__div__NFCI_vol_20d", "Core_CPI_zscore_60d", "VIX_Price_zscore_60d__div__CTAS_Cintas_vol_20d", "Nikkei_Japan_zscore_60d", "XLK_Tech_zscore_60d", "NOC_Northrop_ret_20d", "CLX_Clorox_vol_20d", "vix_zscore_10d__div__heston_var_ev_h7", "Nikkei_Japan_vol_20d", "vix_vol_of_vol_10d__minus__NFCI_vol_20d", "VRP__macross__hmm_p_stress", "HD_zscore_60d", "Industrial_Production_zscore_60d", "NFCI_ret_5d__div__NFCI_vol_20d", "ASX_Australia_vol_20d", "Michigan_Sentiment_ret_20d", "XOM_ret_20d"], "is_new": false}, {"model_id": "v2_h7_GLOBAL_LightGBM_N18", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 7, "n_features": 18, "F1_dir": 0.5909, "F1_UP_FORT": 0.3605, "F1_DOWN_FORT": 0.4519, "train_start": "2001-02-06", "sampler": "SMOTE", "best_params": "{}", "features": ["VIX_Price_zscore_60d__div__NFCI_vol_20d", "Core_CPI_zscore_60d", "VIX_Price_zscore_60d__div__CTAS_Cintas_vol_20d", "Nikkei_Japan_zscore_60d", "XLK_Tech_zscore_60d", "NOC_Northrop_ret_20d", "CLX_Clorox_vol_20d", "vix_zscore_10d__div__heston_var_ev_h7", "Nikkei_Japan_vol_20d", "vix_vol_of_vol_10d__minus__NFCI_vol_20d", "VRP__macross__hmm_p_stress", "HD_zscore_60d", "Industrial_Production_zscore_60d", "NFCI_ret_5d__div__NFCI_vol_20d", "ASX_Australia_vol_20d", "Michigan_Sentiment_ret_20d", "XOM_ret_20d", "VIX_Price_zscore_60d__ret5x__NFCI_zscore_60d"], "is_new": false}, {"model_id": "v2_h7_GLOBAL_LightGBM_N19", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 7, "n_features": 19, "F1_dir": 0.5901, "F1_UP_FORT": 0.373, "F1_DOWN_FORT": 0.4706, "train_start": "2001-02-06", "sampler": "SMOTE", "best_params": "{}", "features": ["VIX_Price_zscore_60d__div__NFCI_vol_20d", "Core_CPI_zscore_60d", "VIX_Price_zscore_60d__div__CTAS_Cintas_vol_20d", "Nikkei_Japan_zscore_60d", "XLK_Tech_zscore_60d", "NOC_Northrop_ret_20d", "CLX_Clorox_vol_20d", "vix_zscore_10d__div__heston_var_ev_h7", "Nikkei_Japan_vol_20d", "vix_vol_of_vol_10d__minus__NFCI_vol_20d", "VRP__macross__hmm_p_stress", "HD_zscore_60d", "Industrial_Production_zscore_60d", "NFCI_ret_5d__div__NFCI_vol_20d", "ASX_Australia_vol_20d", "Michigan_Sentiment_ret_20d", "XOM_ret_20d", "VIX_Price_zscore_60d__ret5x__NFCI_zscore_60d", "VIX_Price_zscore_60d__div__hmm_p_stress"], "is_new": false}, {"model_id": "v2_h7_GLOBAL_LightGBM_N20", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 7, "n_features": 20, "F1_dir": 0.6083, "F1_UP_FORT": 0.3962, "F1_DOWN_FORT": 0.473, "train_start": "2001-02-06", "sampler": "SMOTE", "best_params": "{}", "features": ["VIX_Price_zscore_60d__div__NFCI_vol_20d", "Core_CPI_zscore_60d", "VIX_Price_zscore_60d__div__CTAS_Cintas_vol_20d", "Nikkei_Japan_zscore_60d", "XLK_Tech_zscore_60d", "NOC_Northrop_ret_20d", "CLX_Clorox_vol_20d", "vix_zscore_10d__div__heston_var_ev_h7", "Nikkei_Japan_vol_20d", "vix_vol_of_vol_10d__minus__NFCI_vol_20d", "VRP__macross__hmm_p_stress", "HD_zscore_60d", "Industrial_Production_zscore_60d", "NFCI_ret_5d__div__NFCI_vol_20d", "ASX_Australia_vol_20d", "Michigan_Sentiment_ret_20d", "XOM_ret_20d", "VIX_Price_zscore_60d__ret5x__NFCI_zscore_60d", "VIX_Price_zscore_60d__div__hmm_p_stress", "VIX_Price_zscore_60d__prod__CTAS_Cintas_vol_20d"], "is_new": false}, {"model_id": "v2_h7_GLOBAL_LightGBM_N21", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 7, "n_features": 21, "F1_dir": 0.5983, "F1_UP_FORT": 0.3959, "F1_DOWN_FORT": 0.4737, "train_start": "2001-02-06", "sampler": "SMOTE", "best_params": "{}", "features": ["VIX_Price_zscore_60d__div__NFCI_vol_20d", "Core_CPI_zscore_60d", "VIX_Price_zscore_60d__div__CTAS_Cintas_vol_20d", "Nikkei_Japan_zscore_60d", "XLK_Tech_zscore_60d", "NOC_Northrop_ret_20d", "CLX_Clorox_vol_20d", "vix_zscore_10d__div__heston_var_ev_h7", "Nikkei_Japan_vol_20d", "vix_vol_of_vol_10d__minus__NFCI_vol_20d", "VRP__macross__hmm_p_stress", "HD_zscore_60d", "Industrial_Production_zscore_60d", "NFCI_ret_5d__div__NFCI_vol_20d", "ASX_Australia_vol_20d", "Michigan_Sentiment_ret_20d", "XOM_ret_20d", "VIX_Price_zscore_60d__ret5x__NFCI_zscore_60d", "VIX_Price_zscore_60d__div__hmm_p_stress", "VIX_Price_zscore_60d__prod__CTAS_Cintas_vol_20d", "vix_zscore_10d__div__vix_vol_of_vol_10d"], "is_new": false}, {"model_id": "v2_h7_GLOBAL_LightGBM_N22", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 7, "n_features": 22, "F1_dir": 0.5909, "F1_UP_FORT": 0.3724, "F1_DOWN_FORT": 0.4848, "train_start": "2001-02-06", "sampler": "SMOTE", "best_params": "{}", "features": ["VIX_Price_zscore_60d__div__NFCI_vol_20d", "Core_CPI_zscore_60d", "VIX_Price_zscore_60d__div__CTAS_Cintas_vol_20d", "Nikkei_Japan_zscore_60d", "XLK_Tech_zscore_60d", "NOC_Northrop_ret_20d", "CLX_Clorox_vol_20d", "vix_zscore_10d__div__heston_var_ev_h7", "Nikkei_Japan_vol_20d", "vix_vol_of_vol_10d__minus__NFCI_vol_20d", "VRP__macross__hmm_p_stress", "HD_zscore_60d", "Industrial_Production_zscore_60d", "NFCI_ret_5d__div__NFCI_vol_20d", "ASX_Australia_vol_20d", "Michigan_Sentiment_ret_20d", "XOM_ret_20d", "VIX_Price_zscore_60d__ret5x__NFCI_zscore_60d", "VIX_Price_zscore_60d__div__hmm_p_stress", "VIX_Price_zscore_60d__prod__CTAS_Cintas_vol_20d", "vix_zscore_10d__div__vix_vol_of_vol_10d", "VIX_Price_zscore_60d__zrel__SBUX_ret_5d"], "is_new": false}, {"model_id": "v2_h7_GLOBAL_LightGBM_N23", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 7, "n_features": 23, "F1_dir": 0.6029, "F1_UP_FORT": 0.3899, "F1_DOWN_FORT": 0.4681, "train_start": "2001-02-06", "sampler": "SMOTE", "best_params": "{}", "features": ["VIX_Price_zscore_60d__div__NFCI_vol_20d", "Core_CPI_zscore_60d", "VIX_Price_zscore_60d__div__CTAS_Cintas_vol_20d", "Nikkei_Japan_zscore_60d", "XLK_Tech_zscore_60d", "NOC_Northrop_ret_20d", "CLX_Clorox_vol_20d", "vix_zscore_10d__div__heston_var_ev_h7", "Nikkei_Japan_vol_20d", "vix_vol_of_vol_10d__minus__NFCI_vol_20d", "VRP__macross__hmm_p_stress", "HD_zscore_60d", "Industrial_Production_zscore_60d", "NFCI_ret_5d__div__NFCI_vol_20d", "ASX_Australia_vol_20d", "Michigan_Sentiment_ret_20d", "XOM_ret_20d", "VIX_Price_zscore_60d__ret5x__NFCI_zscore_60d", "VIX_Price_zscore_60d__div__hmm_p_stress", "VIX_Price_zscore_60d__prod__CTAS_Cintas_vol_20d", "vix_zscore_10d__div__vix_vol_of_vol_10d", "VIX_Price_zscore_60d__zrel__SBUX_ret_5d", "VIX_Price_zscore_60d__minus__heston_xi"], "is_new": false}, {"model_id": "v2_h7_GLOBAL_GradientBoosting_N5", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 7, "n_features": 5, "F1_dir": 0.5569, "F1_UP_FORT": 0.2715, "F1_DOWN_FORT": 0.4213, "train_start": "2001-02-06", "sampler": "SMOTE", "best_params": "{}", "features": ["VIX_Price_zscore_60d__div__NFCI_vol_20d", "Core_CPI_zscore_60d", "VIX_Price_zscore_60d__div__CTAS_Cintas_vol_20d", "Nikkei_Japan_zscore_60d", "XLK_Tech_zscore_60d"], "is_new": false}, {"model_id": "v2_h7_GLOBAL_GradientBoosting_N6", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 7, "n_features": 6, "F1_dir": 0.5522, "F1_UP_FORT": 0.2715, "F1_DOWN_FORT": 0.4041, "train_start": "2001-02-06", "sampler": "SMOTE", "best_params": "{}", "features": ["VIX_Price_zscore_60d__div__NFCI_vol_20d", "Core_CPI_zscore_60d", "VIX_Price_zscore_60d__div__CTAS_Cintas_vol_20d", "Nikkei_Japan_zscore_60d", "XLK_Tech_zscore_60d", "NOC_Northrop_ret_20d"], "is_new": false}, {"model_id": "v2_h7_GLOBAL_GradientBoosting_N7", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 7, "n_features": 7, "F1_dir": 0.5183, "F1_UP_FORT": 0.2408, "F1_DOWN_FORT": 0.3744, "train_start": "2001-02-06", "sampler": "SMOTE", "best_params": "{}", "features": ["VIX_Price_zscore_60d__div__NFCI_vol_20d", "Core_CPI_zscore_60d", "VIX_Price_zscore_60d__div__CTAS_Cintas_vol_20d", "Nikkei_Japan_zscore_60d", "XLK_Tech_zscore_60d", "NOC_Northrop_ret_20d", "CLX_Clorox_vol_20d"], "is_new": false}, {"model_id": "v2_h7_GLOBAL_GradientBoosting_N8", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 7, "n_features": 8, "F1_dir": 0.5283, "F1_UP_FORT": 0.2222, "F1_DOWN_FORT": 0.4088, "train_start": "2001-02-06", "sampler": "SMOTE", "best_params": "{}", "features": ["VIX_Price_zscore_60d__div__NFCI_vol_20d", "Core_CPI_zscore_60d", "VIX_Price_zscore_60d__div__CTAS_Cintas_vol_20d", "Nikkei_Japan_zscore_60d", "XLK_Tech_zscore_60d", "NOC_Northrop_ret_20d", "CLX_Clorox_vol_20d", "vix_zscore_10d__div__heston_var_ev_h7"], "is_new": false}, {"model_id": "v2_h7_GLOBAL_GradientBoosting_N9", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 7, "n_features": 9, "F1_dir": 0.5498, "F1_UP_FORT": 0.2969, "F1_DOWN_FORT": 0.4193, "train_start": "2001-02-06", "sampler": "SMOTE", "best_params": "{}", "features": ["VIX_Price_zscore_60d__div__NFCI_vol_20d", "Core_CPI_zscore_60d", "VIX_Price_zscore_60d__div__CTAS_Cintas_vol_20d", "Nikkei_Japan_zscore_60d", "XLK_Tech_zscore_60d", "NOC_Northrop_ret_20d", "CLX_Clorox_vol_20d", "vix_zscore_10d__div__heston_var_ev_h7", "Nikkei_Japan_vol_20d"], "is_new": false}, {"model_id": "v2_h7_GLOBAL_GradientBoosting_N10", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 7, "n_features": 10, "F1_dir": 0.5375, "F1_UP_FORT": 0.2904, "F1_DOWN_FORT": 0.4374, "train_start": "2001-02-06", "sampler": "SMOTE", "best_params": "{}", "features": ["VIX_Price_zscore_60d__div__NFCI_vol_20d", "Core_CPI_zscore_60d", "VIX_Price_zscore_60d__div__CTAS_Cintas_vol_20d", "Nikkei_Japan_zscore_60d", "XLK_Tech_zscore_60d", "NOC_Northrop_ret_20d", "CLX_Clorox_vol_20d", "vix_zscore_10d__div__heston_var_ev_h7", "Nikkei_Japan_vol_20d", "vix_vol_of_vol_10d__minus__NFCI_vol_20d"], "is_new": false}, {"model_id": "v2_h7_GLOBAL_GradientBoosting_N11", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 7, "n_features": 11, "F1_dir": 0.554, "F1_UP_FORT": 0.2982, "F1_DOWN_FORT": 0.4402, "train_start": "2001-02-06", "sampler": "SMOTE", "best_params": "{}", "features": ["VIX_Price_zscore_60d__div__NFCI_vol_20d", "Core_CPI_zscore_60d", "VIX_Price_zscore_60d__div__CTAS_Cintas_vol_20d", "Nikkei_Japan_zscore_60d", "XLK_Tech_zscore_60d", "NOC_Northrop_ret_20d", "CLX_Clorox_vol_20d", "vix_zscore_10d__div__heston_var_ev_h7", "Nikkei_Japan_vol_20d", "vix_vol_of_vol_10d__minus__NFCI_vol_20d", "VRP__macross__hmm_p_stress"], "is_new": false}, {"model_id": "v2_h7_GLOBAL_GradientBoosting_N12", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 7, "n_features": 12, "F1_dir": 0.5449, "F1_UP_FORT": 0.3027, "F1_DOWN_FORT": 0.455, "train_start": "2001-02-06", "sampler": "SMOTE", "best_params": "{}", "features": ["VIX_Price_zscore_60d__div__NFCI_vol_20d", "Core_CPI_zscore_60d", "VIX_Price_zscore_60d__div__CTAS_Cintas_vol_20d", "Nikkei_Japan_zscore_60d", "XLK_Tech_zscore_60d", "NOC_Northrop_ret_20d", "CLX_Clorox_vol_20d", "vix_zscore_10d__div__heston_var_ev_h7", "Nikkei_Japan_vol_20d", "vix_vol_of_vol_10d__minus__NFCI_vol_20d", "VRP__macross__hmm_p_stress", "HD_zscore_60d"], "is_new": false}, {"model_id": "v2_h7_GLOBAL_GradientBoosting_N13", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 7, "n_features": 13, "F1_dir": 0.5234, "F1_UP_FORT": 0.2897, "F1_DOWN_FORT": 0.4406, "train_start": "2001-02-06", "sampler": "SMOTE", "best_params": "{}", "features": ["VIX_Price_zscore_60d__div__NFCI_vol_20d", "Core_CPI_zscore_60d", "VIX_Price_zscore_60d__div__CTAS_Cintas_vol_20d", "Nikkei_Japan_zscore_60d", "XLK_Tech_zscore_60d", "NOC_Northrop_ret_20d", "CLX_Clorox_vol_20d", "vix_zscore_10d__div__heston_var_ev_h7", "Nikkei_Japan_vol_20d", "vix_vol_of_vol_10d__minus__NFCI_vol_20d", "VRP__macross__hmm_p_stress", "HD_zscore_60d", "Industrial_Production_zscore_60d"], "is_new": false}, {"model_id": "v2_h7_GLOBAL_GradientBoosting_N14", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 7, "n_features": 14, "F1_dir": 0.5743, "F1_UP_FORT": 0.3233, "F1_DOWN_FORT": 0.4634, "train_start": "2001-02-06", "sampler": "SMOTE", "best_params": "{}", "features": ["VIX_Price_zscore_60d__div__NFCI_vol_20d", "Core_CPI_zscore_60d", "VIX_Price_zscore_60d__div__CTAS_Cintas_vol_20d", "Nikkei_Japan_zscore_60d", "XLK_Tech_zscore_60d", "NOC_Northrop_ret_20d", "CLX_Clorox_vol_20d", "vix_zscore_10d__div__heston_var_ev_h7", "Nikkei_Japan_vol_20d", "vix_vol_of_vol_10d__minus__NFCI_vol_20d", "VRP__macross__hmm_p_stress", "HD_zscore_60d", "Industrial_Production_zscore_60d", "NFCI_ret_5d__div__NFCI_vol_20d"], "is_new": false}, {"model_id": "v2_h7_GLOBAL_GradientBoosting_N15", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 7, "n_features": 15, "F1_dir": 0.5807, "F1_UP_FORT": 0.36, "F1_DOWN_FORT": 0.4415, "train_start": "2001-02-06", "sampler": "SMOTE", "best_params": "{}", "features": ["VIX_Price_zscore_60d__div__NFCI_vol_20d", "Core_CPI_zscore_60d", "VIX_Price_zscore_60d__div__CTAS_Cintas_vol_20d", "Nikkei_Japan_zscore_60d", "XLK_Tech_zscore_60d", "NOC_Northrop_ret_20d", "CLX_Clorox_vol_20d", "vix_zscore_10d__div__heston_var_ev_h7", "Nikkei_Japan_vol_20d", "vix_vol_of_vol_10d__minus__NFCI_vol_20d", "VRP__macross__hmm_p_stress", "HD_zscore_60d", "Industrial_Production_zscore_60d", "NFCI_ret_5d__div__NFCI_vol_20d", "ASX_Australia_vol_20d"], "is_new": false}, {"model_id": "v2_h7_GLOBAL_GradientBoosting_N16", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 7, "n_features": 16, "F1_dir": 0.5915, "F1_UP_FORT": 0.3673, "F1_DOWN_FORT": 0.4568, "train_start": "2001-02-06", "sampler": "SMOTE", "best_params": "{}", "features": ["VIX_Price_zscore_60d__div__NFCI_vol_20d", "Core_CPI_zscore_60d", "VIX_Price_zscore_60d__div__CTAS_Cintas_vol_20d", "Nikkei_Japan_zscore_60d", "XLK_Tech_zscore_60d", "NOC_Northrop_ret_20d", "CLX_Clorox_vol_20d", "vix_zscore_10d__div__heston_var_ev_h7", "Nikkei_Japan_vol_20d", "vix_vol_of_vol_10d__minus__NFCI_vol_20d", "VRP__macross__hmm_p_stress", "HD_zscore_60d", "Industrial_Production_zscore_60d", "NFCI_ret_5d__div__NFCI_vol_20d", "ASX_Australia_vol_20d", "Michigan_Sentiment_ret_20d"], "is_new": false}, {"model_id": "v2_h7_GLOBAL_GradientBoosting_N17", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 7, "n_features": 17, "F1_dir": 0.593, "F1_UP_FORT": 0.3902, "F1_DOWN_FORT": 0.4769, "train_start": "2001-02-06", "sampler": "SMOTE", "best_params": "{}", "features": ["VIX_Price_zscore_60d__div__NFCI_vol_20d", "Core_CPI_zscore_60d", "VIX_Price_zscore_60d__div__CTAS_Cintas_vol_20d", "Nikkei_Japan_zscore_60d", "XLK_Tech_zscore_60d", "NOC_Northrop_ret_20d", "CLX_Clorox_vol_20d", "vix_zscore_10d__div__heston_var_ev_h7", "Nikkei_Japan_vol_20d", "vix_vol_of_vol_10d__minus__NFCI_vol_20d", "VRP__macross__hmm_p_stress", "HD_zscore_60d", "Industrial_Production_zscore_60d", "NFCI_ret_5d__div__NFCI_vol_20d", "ASX_Australia_vol_20d", "Michigan_Sentiment_ret_20d", "XOM_ret_20d"], "is_new": false}, {"model_id": "v2_h7_GLOBAL_GradientBoosting_N18", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 7, "n_features": 18, "F1_dir": 0.5833, "F1_UP_FORT": 0.3614, "F1_DOWN_FORT": 0.441, "train_start": "2001-02-06", "sampler": "SMOTE", "best_params": "{}", "features": ["VIX_Price_zscore_60d__div__NFCI_vol_20d", "Core_CPI_zscore_60d", "VIX_Price_zscore_60d__div__CTAS_Cintas_vol_20d", "Nikkei_Japan_zscore_60d", "XLK_Tech_zscore_60d", "NOC_Northrop_ret_20d", "CLX_Clorox_vol_20d", "vix_zscore_10d__div__heston_var_ev_h7", "Nikkei_Japan_vol_20d", "vix_vol_of_vol_10d__minus__NFCI_vol_20d", "VRP__macross__hmm_p_stress", "HD_zscore_60d", "Industrial_Production_zscore_60d", "NFCI_ret_5d__div__NFCI_vol_20d", "ASX_Australia_vol_20d", "Michigan_Sentiment_ret_20d", "XOM_ret_20d", "VIX_Price_zscore_60d__ret5x__NFCI_zscore_60d"], "is_new": false}, {"model_id": "v2_h7_GLOBAL_GradientBoosting_N19", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 7, "n_features": 19, "F1_dir": 0.59, "F1_UP_FORT": 0.3848, "F1_DOWN_FORT": 0.4602, "train_start": "2001-02-06", "sampler": "SMOTE", "best_params": "{}", "features": ["VIX_Price_zscore_60d__div__NFCI_vol_20d", "Core_CPI_zscore_60d", "VIX_Price_zscore_60d__div__CTAS_Cintas_vol_20d", "Nikkei_Japan_zscore_60d", "XLK_Tech_zscore_60d", "NOC_Northrop_ret_20d", "CLX_Clorox_vol_20d", "vix_zscore_10d__div__heston_var_ev_h7", "Nikkei_Japan_vol_20d", "vix_vol_of_vol_10d__minus__NFCI_vol_20d", "VRP__macross__hmm_p_stress", "HD_zscore_60d", "Industrial_Production_zscore_60d", "NFCI_ret_5d__div__NFCI_vol_20d", "ASX_Australia_vol_20d", "Michigan_Sentiment_ret_20d", "XOM_ret_20d", "VIX_Price_zscore_60d__ret5x__NFCI_zscore_60d", "VIX_Price_zscore_60d__div__hmm_p_stress"], "is_new": false}, {"model_id": "v2_h7_GLOBAL_GradientBoosting_N20", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 7, "n_features": 20, "F1_dir": 0.5961, "F1_UP_FORT": 0.3884, "F1_DOWN_FORT": 0.4778, "train_start": "2001-02-06", "sampler": "SMOTE", "best_params": "{}", "features": ["VIX_Price_zscore_60d__div__NFCI_vol_20d", "Core_CPI_zscore_60d", "VIX_Price_zscore_60d__div__CTAS_Cintas_vol_20d", "Nikkei_Japan_zscore_60d", "XLK_Tech_zscore_60d", "NOC_Northrop_ret_20d", "CLX_Clorox_vol_20d", "vix_zscore_10d__div__heston_var_ev_h7", "Nikkei_Japan_vol_20d", "vix_vol_of_vol_10d__minus__NFCI_vol_20d", "VRP__macross__hmm_p_stress", "HD_zscore_60d", "Industrial_Production_zscore_60d", "NFCI_ret_5d__div__NFCI_vol_20d", "ASX_Australia_vol_20d", "Michigan_Sentiment_ret_20d", "XOM_ret_20d", "VIX_Price_zscore_60d__ret5x__NFCI_zscore_60d", "VIX_Price_zscore_60d__div__hmm_p_stress", "VIX_Price_zscore_60d__prod__CTAS_Cintas_vol_20d"], "is_new": false}, {"model_id": "v2_h7_GLOBAL_GradientBoosting_N21", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 7, "n_features": 21, "F1_dir": 0.5871, "F1_UP_FORT": 0.3909, "F1_DOWN_FORT": 0.4986, "train_start": "2001-02-06", "sampler": "SMOTE", "best_params": "{}", "features": ["VIX_Price_zscore_60d__div__NFCI_vol_20d", "Core_CPI_zscore_60d", "VIX_Price_zscore_60d__div__CTAS_Cintas_vol_20d", "Nikkei_Japan_zscore_60d", "XLK_Tech_zscore_60d", "NOC_Northrop_ret_20d", "CLX_Clorox_vol_20d", "vix_zscore_10d__div__heston_var_ev_h7", "Nikkei_Japan_vol_20d", "vix_vol_of_vol_10d__minus__NFCI_vol_20d", "VRP__macross__hmm_p_stress", "HD_zscore_60d", "Industrial_Production_zscore_60d", "NFCI_ret_5d__div__NFCI_vol_20d", "ASX_Australia_vol_20d", "Michigan_Sentiment_ret_20d", "XOM_ret_20d", "VIX_Price_zscore_60d__ret5x__NFCI_zscore_60d", "VIX_Price_zscore_60d__div__hmm_p_stress", "VIX_Price_zscore_60d__prod__CTAS_Cintas_vol_20d", "vix_zscore_10d__div__vix_vol_of_vol_10d"], "is_new": false}, {"model_id": "v2_h7_GLOBAL_GradientBoosting_N22", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 7, "n_features": 22, "F1_dir": 0.5908, "F1_UP_FORT": 0.3822, "F1_DOWN_FORT": 0.4848, "train_start": "2001-02-06", "sampler": "SMOTE", "best_params": "{}", "features": ["VIX_Price_zscore_60d__div__NFCI_vol_20d", "Core_CPI_zscore_60d", "VIX_Price_zscore_60d__div__CTAS_Cintas_vol_20d", "Nikkei_Japan_zscore_60d", "XLK_Tech_zscore_60d", "NOC_Northrop_ret_20d", "CLX_Clorox_vol_20d", "vix_zscore_10d__div__heston_var_ev_h7", "Nikkei_Japan_vol_20d", "vix_vol_of_vol_10d__minus__NFCI_vol_20d", "VRP__macross__hmm_p_stress", "HD_zscore_60d", "Industrial_Production_zscore_60d", "NFCI_ret_5d__div__NFCI_vol_20d", "ASX_Australia_vol_20d", "Michigan_Sentiment_ret_20d", "XOM_ret_20d", "VIX_Price_zscore_60d__ret5x__NFCI_zscore_60d", "VIX_Price_zscore_60d__div__hmm_p_stress", "VIX_Price_zscore_60d__prod__CTAS_Cintas_vol_20d", "vix_zscore_10d__div__vix_vol_of_vol_10d", "VIX_Price_zscore_60d__zrel__SBUX_ret_5d"], "is_new": false}, {"model_id": "v2_h7_GLOBAL_GradientBoosting_N23", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 7, "n_features": 23, "F1_dir": 0.6007, "F1_UP_FORT": 0.3745, "F1_DOWN_FORT": 0.4765, "train_start": "2001-02-06", "sampler": "SMOTE", "best_params": "{}", "features": ["VIX_Price_zscore_60d__div__NFCI_vol_20d", "Core_CPI_zscore_60d", "VIX_Price_zscore_60d__div__CTAS_Cintas_vol_20d", "Nikkei_Japan_zscore_60d", "XLK_Tech_zscore_60d", "NOC_Northrop_ret_20d", "CLX_Clorox_vol_20d", "vix_zscore_10d__div__heston_var_ev_h7", "Nikkei_Japan_vol_20d", "vix_vol_of_vol_10d__minus__NFCI_vol_20d", "VRP__macross__hmm_p_stress", "HD_zscore_60d", "Industrial_Production_zscore_60d", "NFCI_ret_5d__div__NFCI_vol_20d", "ASX_Australia_vol_20d", "Michigan_Sentiment_ret_20d", "XOM_ret_20d", "VIX_Price_zscore_60d__ret5x__NFCI_zscore_60d", "VIX_Price_zscore_60d__div__hmm_p_stress", "VIX_Price_zscore_60d__prod__CTAS_Cintas_vol_20d", "vix_zscore_10d__div__vix_vol_of_vol_10d", "VIX_Price_zscore_60d__zrel__SBUX_ret_5d", "VIX_Price_zscore_60d__minus__heston_xi"], "is_new": false}, {"model_id": "v2_h7_GLOBAL_RandomForest_N5", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 7, "n_features": 5, "F1_dir": 0.5741, "F1_UP_FORT": 0.1924, "F1_DOWN_FORT": 0.4601, "train_start": "2001-02-06", "sampler": "SMOTE", "best_params": "{}", "features": ["VIX_Price_zscore_60d__div__NFCI_vol_20d", "Core_CPI_zscore_60d", "VIX_Price_zscore_60d__div__CTAS_Cintas_vol_20d", "Nikkei_Japan_zscore_60d", "XLK_Tech_zscore_60d"], "is_new": false}, {"model_id": "v2_h7_GLOBAL_RandomForest_N6", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 7, "n_features": 6, "F1_dir": 0.5616, "F1_UP_FORT": 0.2014, "F1_DOWN_FORT": 0.4733, "train_start": "2001-02-06", "sampler": "SMOTE", "best_params": "{}", "features": ["VIX_Price_zscore_60d__div__NFCI_vol_20d", "Core_CPI_zscore_60d", "VIX_Price_zscore_60d__div__CTAS_Cintas_vol_20d", "Nikkei_Japan_zscore_60d", "XLK_Tech_zscore_60d", "NOC_Northrop_ret_20d"], "is_new": false}, {"model_id": "v2_h7_GLOBAL_RandomForest_N7", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 7, "n_features": 7, "F1_dir": 0.5488, "F1_UP_FORT": 0.2155, "F1_DOWN_FORT": 0.4378, "train_start": "2001-02-06", "sampler": "SMOTE", "best_params": "{}", "features": ["VIX_Price_zscore_60d__div__NFCI_vol_20d", "Core_CPI_zscore_60d", "VIX_Price_zscore_60d__div__CTAS_Cintas_vol_20d", "Nikkei_Japan_zscore_60d", "XLK_Tech_zscore_60d", "NOC_Northrop_ret_20d", "CLX_Clorox_vol_20d"], "is_new": false}, {"model_id": "v2_h7_GLOBAL_RandomForest_N8", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 7, "n_features": 8, "F1_dir": 0.5567, "F1_UP_FORT": 0.1827, "F1_DOWN_FORT": 0.4601, "train_start": "2001-02-06", "sampler": "SMOTE", "best_params": "{}", "features": ["VIX_Price_zscore_60d__div__NFCI_vol_20d", "Core_CPI_zscore_60d", "VIX_Price_zscore_60d__div__CTAS_Cintas_vol_20d", "Nikkei_Japan_zscore_60d", "XLK_Tech_zscore_60d", "NOC_Northrop_ret_20d", "CLX_Clorox_vol_20d", "vix_zscore_10d__div__heston_var_ev_h7"], "is_new": false}, {"model_id": "v2_h7_GLOBAL_RandomForest_N9", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 7, "n_features": 9, "F1_dir": 0.5686, "F1_UP_FORT": 0.2305, "F1_DOWN_FORT": 0.4739, "train_start": "2001-02-06", "sampler": "SMOTE", "best_params": "{}", "features": ["VIX_Price_zscore_60d__div__NFCI_vol_20d", "Core_CPI_zscore_60d", "VIX_Price_zscore_60d__div__CTAS_Cintas_vol_20d", "Nikkei_Japan_zscore_60d", "XLK_Tech_zscore_60d", "NOC_Northrop_ret_20d", "CLX_Clorox_vol_20d", "vix_zscore_10d__div__heston_var_ev_h7", "Nikkei_Japan_vol_20d"], "is_new": false}, {"model_id": "v2_h7_GLOBAL_RandomForest_N10", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 7, "n_features": 10, "F1_dir": 0.573, "F1_UP_FORT": 0.2633, "F1_DOWN_FORT": 0.4861, "train_start": "2001-02-06", "sampler": "SMOTE", "best_params": "{}", "features": ["VIX_Price_zscore_60d__div__NFCI_vol_20d", "Core_CPI_zscore_60d", "VIX_Price_zscore_60d__div__CTAS_Cintas_vol_20d", "Nikkei_Japan_zscore_60d", "XLK_Tech_zscore_60d", "NOC_Northrop_ret_20d", "CLX_Clorox_vol_20d", "vix_zscore_10d__div__heston_var_ev_h7", "Nikkei_Japan_vol_20d", "vix_vol_of_vol_10d__minus__NFCI_vol_20d"], "is_new": false}, {"model_id": "v2_h7_GLOBAL_RandomForest_N11", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 7, "n_features": 11, "F1_dir": 0.5689, "F1_UP_FORT": 0.2139, "F1_DOWN_FORT": 0.4665, "train_start": "2001-02-06", "sampler": "SMOTE", "best_params": "{}", "features": ["VIX_Price_zscore_60d__div__NFCI_vol_20d", "Core_CPI_zscore_60d", "VIX_Price_zscore_60d__div__CTAS_Cintas_vol_20d", "Nikkei_Japan_zscore_60d", "XLK_Tech_zscore_60d", "NOC_Northrop_ret_20d", "CLX_Clorox_vol_20d", "vix_zscore_10d__div__heston_var_ev_h7", "Nikkei_Japan_vol_20d", "vix_vol_of_vol_10d__minus__NFCI_vol_20d", "VRP__macross__hmm_p_stress"], "is_new": false}, {"model_id": "v2_h7_GLOBAL_RandomForest_N12", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 7, "n_features": 12, "F1_dir": 0.5706, "F1_UP_FORT": 0.2277, "F1_DOWN_FORT": 0.4647, "train_start": "2001-02-06", "sampler": "SMOTE", "best_params": "{}", "features": ["VIX_Price_zscore_60d__div__NFCI_vol_20d", "Core_CPI_zscore_60d", "VIX_Price_zscore_60d__div__CTAS_Cintas_vol_20d", "Nikkei_Japan_zscore_60d", "XLK_Tech_zscore_60d", "NOC_Northrop_ret_20d", "CLX_Clorox_vol_20d", "vix_zscore_10d__div__heston_var_ev_h7", "Nikkei_Japan_vol_20d", "vix_vol_of_vol_10d__minus__NFCI_vol_20d", "VRP__macross__hmm_p_stress", "HD_zscore_60d"], "is_new": false}, {"model_id": "v2_h7_GLOBAL_RandomForest_N13", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 7, "n_features": 13, "F1_dir": 0.5624, "F1_UP_FORT": 0.1966, "F1_DOWN_FORT": 0.4686, "train_start": "2001-02-06", "sampler": "SMOTE", "best_params": "{}", "features": ["VIX_Price_zscore_60d__div__NFCI_vol_20d", "Core_CPI_zscore_60d", "VIX_Price_zscore_60d__div__CTAS_Cintas_vol_20d", "Nikkei_Japan_zscore_60d", "XLK_Tech_zscore_60d", "NOC_Northrop_ret_20d", "CLX_Clorox_vol_20d", "vix_zscore_10d__div__heston_var_ev_h7", "Nikkei_Japan_vol_20d", "vix_vol_of_vol_10d__minus__NFCI_vol_20d", "VRP__macross__hmm_p_stress", "HD_zscore_60d", "Industrial_Production_zscore_60d"], "is_new": false}, {"model_id": "v2_h7_GLOBAL_RandomForest_N14", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 7, "n_features": 14, "F1_dir": 0.6168, "F1_UP_FORT": 0.358, "F1_DOWN_FORT": 0.4911, "train_start": "2001-02-06", "sampler": "SMOTE", "best_params": "{}", "features": ["VIX_Price_zscore_60d__div__NFCI_vol_20d", "Core_CPI_zscore_60d", "VIX_Price_zscore_60d__div__CTAS_Cintas_vol_20d", "Nikkei_Japan_zscore_60d", "XLK_Tech_zscore_60d", "NOC_Northrop_ret_20d", "CLX_Clorox_vol_20d", "vix_zscore_10d__div__heston_var_ev_h7", "Nikkei_Japan_vol_20d", "vix_vol_of_vol_10d__minus__NFCI_vol_20d", "VRP__macross__hmm_p_stress", "HD_zscore_60d", "Industrial_Production_zscore_60d", "NFCI_ret_5d__div__NFCI_vol_20d"], "is_new": false}, {"model_id": "v2_h7_GLOBAL_RandomForest_N15", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 7, "n_features": 15, "F1_dir": 0.6205, "F1_UP_FORT": 0.3686, "F1_DOWN_FORT": 0.4918, "train_start": "2001-02-06", "sampler": "SMOTE", "best_params": "{}", "features": ["VIX_Price_zscore_60d__div__NFCI_vol_20d", "Core_CPI_zscore_60d", "VIX_Price_zscore_60d__div__CTAS_Cintas_vol_20d", "Nikkei_Japan_zscore_60d", "XLK_Tech_zscore_60d", "NOC_Northrop_ret_20d", "CLX_Clorox_vol_20d", "vix_zscore_10d__div__heston_var_ev_h7", "Nikkei_Japan_vol_20d", "vix_vol_of_vol_10d__minus__NFCI_vol_20d", "VRP__macross__hmm_p_stress", "HD_zscore_60d", "Industrial_Production_zscore_60d", "NFCI_ret_5d__div__NFCI_vol_20d", "ASX_Australia_vol_20d"], "is_new": false}, {"model_id": "v2_h7_GLOBAL_RandomForest_N16", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 7, "n_features": 16, "F1_dir": 0.6307, "F1_UP_FORT": 0.3951, "F1_DOWN_FORT": 0.4944, "train_start": "2001-02-06", "sampler": "SMOTE", "best_params": "{}", "features": ["VIX_Price_zscore_60d__div__NFCI_vol_20d", "Core_CPI_zscore_60d", "VIX_Price_zscore_60d__div__CTAS_Cintas_vol_20d", "Nikkei_Japan_zscore_60d", "XLK_Tech_zscore_60d", "NOC_Northrop_ret_20d", "CLX_Clorox_vol_20d", "vix_zscore_10d__div__heston_var_ev_h7", "Nikkei_Japan_vol_20d", "vix_vol_of_vol_10d__minus__NFCI_vol_20d", "VRP__macross__hmm_p_stress", "HD_zscore_60d", "Industrial_Production_zscore_60d", "NFCI_ret_5d__div__NFCI_vol_20d", "ASX_Australia_vol_20d", "Michigan_Sentiment_ret_20d"], "is_new": false}, {"model_id": "v2_h7_GLOBAL_RandomForest_N17", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 7, "n_features": 17, "F1_dir": 0.6262, "F1_UP_FORT": 0.3978, "F1_DOWN_FORT": 0.49, "train_start": "2001-02-06", "sampler": "SMOTE", "best_params": "{}", "features": ["VIX_Price_zscore_60d__div__NFCI_vol_20d", "Core_CPI_zscore_60d", "VIX_Price_zscore_60d__div__CTAS_Cintas_vol_20d", "Nikkei_Japan_zscore_60d", "XLK_Tech_zscore_60d", "NOC_Northrop_ret_20d", "CLX_Clorox_vol_20d", "vix_zscore_10d__div__heston_var_ev_h7", "Nikkei_Japan_vol_20d", "vix_vol_of_vol_10d__minus__NFCI_vol_20d", "VRP__macross__hmm_p_stress", "HD_zscore_60d", "Industrial_Production_zscore_60d", "NFCI_ret_5d__div__NFCI_vol_20d", "ASX_Australia_vol_20d", "Michigan_Sentiment_ret_20d", "XOM_ret_20d"], "is_new": false}, {"model_id": "v2_h7_GLOBAL_RandomForest_N18", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 7, "n_features": 18, "F1_dir": 0.6236, "F1_UP_FORT": 0.4006, "F1_DOWN_FORT": 0.4979, "train_start": "2001-02-06", "sampler": "SMOTE", "best_params": "{}", "features": ["VIX_Price_zscore_60d__div__NFCI_vol_20d", "Core_CPI_zscore_60d", "VIX_Price_zscore_60d__div__CTAS_Cintas_vol_20d", "Nikkei_Japan_zscore_60d", "XLK_Tech_zscore_60d", "NOC_Northrop_ret_20d", "CLX_Clorox_vol_20d", "vix_zscore_10d__div__heston_var_ev_h7", "Nikkei_Japan_vol_20d", "vix_vol_of_vol_10d__minus__NFCI_vol_20d", "VRP__macross__hmm_p_stress", "HD_zscore_60d", "Industrial_Production_zscore_60d", "NFCI_ret_5d__div__NFCI_vol_20d", "ASX_Australia_vol_20d", "Michigan_Sentiment_ret_20d", "XOM_ret_20d", "VIX_Price_zscore_60d__ret5x__NFCI_zscore_60d"], "is_new": false}, {"model_id": "v2_h7_GLOBAL_RandomForest_N19", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 7, "n_features": 19, "F1_dir": 0.6289, "F1_UP_FORT": 0.3636, "F1_DOWN_FORT": 0.4965, "train_start": "2001-02-06", "sampler": "SMOTE", "best_params": "{}", "features": ["VIX_Price_zscore_60d__div__NFCI_vol_20d", "Core_CPI_zscore_60d", "VIX_Price_zscore_60d__div__CTAS_Cintas_vol_20d", "Nikkei_Japan_zscore_60d", "XLK_Tech_zscore_60d", "NOC_Northrop_ret_20d", "CLX_Clorox_vol_20d", "vix_zscore_10d__div__heston_var_ev_h7", "Nikkei_Japan_vol_20d", "vix_vol_of_vol_10d__minus__NFCI_vol_20d", "VRP__macross__hmm_p_stress", "HD_zscore_60d", "Industrial_Production_zscore_60d", "NFCI_ret_5d__div__NFCI_vol_20d", "ASX_Australia_vol_20d", "Michigan_Sentiment_ret_20d", "XOM_ret_20d", "VIX_Price_zscore_60d__ret5x__NFCI_zscore_60d", "VIX_Price_zscore_60d__div__hmm_p_stress"], "is_new": false}, {"model_id": "v2_h7_GLOBAL_RandomForest_N20", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 7, "n_features": 20, "F1_dir": 0.6262, "F1_UP_FORT": 0.3652, "F1_DOWN_FORT": 0.5075, "train_start": "2001-02-06", "sampler": "SMOTE", "best_params": "{}", "features": ["VIX_Price_zscore_60d__div__NFCI_vol_20d", "Core_CPI_zscore_60d", "VIX_Price_zscore_60d__div__CTAS_Cintas_vol_20d", "Nikkei_Japan_zscore_60d", "XLK_Tech_zscore_60d", "NOC_Northrop_ret_20d", "CLX_Clorox_vol_20d", "vix_zscore_10d__div__heston_var_ev_h7", "Nikkei_Japan_vol_20d", "vix_vol_of_vol_10d__minus__NFCI_vol_20d", "VRP__macross__hmm_p_stress", "HD_zscore_60d", "Industrial_Production_zscore_60d", "NFCI_ret_5d__div__NFCI_vol_20d", "ASX_Australia_vol_20d", "Michigan_Sentiment_ret_20d", "XOM_ret_20d", "VIX_Price_zscore_60d__ret5x__NFCI_zscore_60d", "VIX_Price_zscore_60d__div__hmm_p_stress", "VIX_Price_zscore_60d__prod__CTAS_Cintas_vol_20d"], "is_new": false}, {"model_id": "v2_h7_GLOBAL_RandomForest_N21", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 7, "n_features": 21, "F1_dir": 0.6201, "F1_UP_FORT": 0.3536, "F1_DOWN_FORT": 0.5083, "train_start": "2001-02-06", "sampler": "SMOTE", "best_params": "{}", "features": ["VIX_Price_zscore_60d__div__NFCI_vol_20d", "Core_CPI_zscore_60d", "VIX_Price_zscore_60d__div__CTAS_Cintas_vol_20d", "Nikkei_Japan_zscore_60d", "XLK_Tech_zscore_60d", "NOC_Northrop_ret_20d", "CLX_Clorox_vol_20d", "vix_zscore_10d__div__heston_var_ev_h7", "Nikkei_Japan_vol_20d", "vix_vol_of_vol_10d__minus__NFCI_vol_20d", "VRP__macross__hmm_p_stress", "HD_zscore_60d", "Industrial_Production_zscore_60d", "NFCI_ret_5d__div__NFCI_vol_20d", "ASX_Australia_vol_20d", "Michigan_Sentiment_ret_20d", "XOM_ret_20d", "VIX_Price_zscore_60d__ret5x__NFCI_zscore_60d", "VIX_Price_zscore_60d__div__hmm_p_stress", "VIX_Price_zscore_60d__prod__CTAS_Cintas_vol_20d", "vix_zscore_10d__div__vix_vol_of_vol_10d"], "is_new": false}, {"model_id": "v2_h7_GLOBAL_RandomForest_N22", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 7, "n_features": 22, "F1_dir": 0.63, "F1_UP_FORT": 0.3639, "F1_DOWN_FORT": 0.5117, "train_start": "2001-02-06", "sampler": "SMOTE", "best_params": "{}", "features": ["VIX_Price_zscore_60d__div__NFCI_vol_20d", "Core_CPI_zscore_60d", "VIX_Price_zscore_60d__div__CTAS_Cintas_vol_20d", "Nikkei_Japan_zscore_60d", "XLK_Tech_zscore_60d", "NOC_Northrop_ret_20d", "CLX_Clorox_vol_20d", "vix_zscore_10d__div__heston_var_ev_h7", "Nikkei_Japan_vol_20d", "vix_vol_of_vol_10d__minus__NFCI_vol_20d", "VRP__macross__hmm_p_stress", "HD_zscore_60d", "Industrial_Production_zscore_60d", "NFCI_ret_5d__div__NFCI_vol_20d", "ASX_Australia_vol_20d", "Michigan_Sentiment_ret_20d", "XOM_ret_20d", "VIX_Price_zscore_60d__ret5x__NFCI_zscore_60d", "VIX_Price_zscore_60d__div__hmm_p_stress", "VIX_Price_zscore_60d__prod__CTAS_Cintas_vol_20d", "vix_zscore_10d__div__vix_vol_of_vol_10d", "VIX_Price_zscore_60d__zrel__SBUX_ret_5d"], "is_new": false}, {"model_id": "v2_h7_GLOBAL_RandomForest_N23", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 7, "n_features": 23, "F1_dir": 0.6252, "F1_UP_FORT": 0.3591, "F1_DOWN_FORT": 0.5061, "train_start": "2001-02-06", "sampler": "SMOTE", "best_params": "{}", "features": ["VIX_Price_zscore_60d__div__NFCI_vol_20d", "Core_CPI_zscore_60d", "VIX_Price_zscore_60d__div__CTAS_Cintas_vol_20d", "Nikkei_Japan_zscore_60d", "XLK_Tech_zscore_60d", "NOC_Northrop_ret_20d", "CLX_Clorox_vol_20d", "vix_zscore_10d__div__heston_var_ev_h7", "Nikkei_Japan_vol_20d", "vix_vol_of_vol_10d__minus__NFCI_vol_20d", "VRP__macross__hmm_p_stress", "HD_zscore_60d", "Industrial_Production_zscore_60d", "NFCI_ret_5d__div__NFCI_vol_20d", "ASX_Australia_vol_20d", "Michigan_Sentiment_ret_20d", "XOM_ret_20d", "VIX_Price_zscore_60d__ret5x__NFCI_zscore_60d", "VIX_Price_zscore_60d__div__hmm_p_stress", "VIX_Price_zscore_60d__prod__CTAS_Cintas_vol_20d", "vix_zscore_10d__div__vix_vol_of_vol_10d", "VIX_Price_zscore_60d__zrel__SBUX_ret_5d", "VIX_Price_zscore_60d__minus__heston_xi"], "is_new": false}, {"model_id": "v2_h7_GLOBAL_LogisticRegression_N5", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 7, "n_features": 5, "F1_dir": 0.592, "F1_UP_FORT": 0.1191, "F1_DOWN_FORT": 0.4663, "train_start": "2001-02-06", "sampler": "SMOTE", "best_params": "{}", "features": ["VIX_Price_zscore_60d__div__NFCI_vol_20d", "Core_CPI_zscore_60d", "VIX_Price_zscore_60d__div__CTAS_Cintas_vol_20d", "Nikkei_Japan_zscore_60d", "XLK_Tech_zscore_60d"], "is_new": false}, {"model_id": "v2_h7_GLOBAL_LogisticRegression_N6", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 7, "n_features": 6, "F1_dir": 0.5949, "F1_UP_FORT": 0.2031, "F1_DOWN_FORT": 0.4389, "train_start": "2001-02-06", "sampler": "SMOTE", "best_params": "{}", "features": ["VIX_Price_zscore_60d__div__NFCI_vol_20d", "Core_CPI_zscore_60d", "VIX_Price_zscore_60d__div__CTAS_Cintas_vol_20d", "Nikkei_Japan_zscore_60d", "XLK_Tech_zscore_60d", "NOC_Northrop_ret_20d"], "is_new": false}, {"model_id": "v2_h7_GLOBAL_LogisticRegression_N7", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 7, "n_features": 7, "F1_dir": 0.5702, "F1_UP_FORT": 0.2932, "F1_DOWN_FORT": 0.4296, "train_start": "2001-02-06", "sampler": "SMOTE", "best_params": "{}", "features": ["VIX_Price_zscore_60d__div__NFCI_vol_20d", "Core_CPI_zscore_60d", "VIX_Price_zscore_60d__div__CTAS_Cintas_vol_20d", "Nikkei_Japan_zscore_60d", "XLK_Tech_zscore_60d", "NOC_Northrop_ret_20d", "CLX_Clorox_vol_20d"], "is_new": false}, {"model_id": "v2_h7_GLOBAL_LogisticRegression_N8", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 7, "n_features": 8, "F1_dir": 0.5696, "F1_UP_FORT": 0.2721, "F1_DOWN_FORT": 0.435, "train_start": "2001-02-06", "sampler": "SMOTE", "best_params": "{}", "features": ["VIX_Price_zscore_60d__div__NFCI_vol_20d", "Core_CPI_zscore_60d", "VIX_Price_zscore_60d__div__CTAS_Cintas_vol_20d", "Nikkei_Japan_zscore_60d", "XLK_Tech_zscore_60d", "NOC_Northrop_ret_20d", "CLX_Clorox_vol_20d", "vix_zscore_10d__div__heston_var_ev_h7"], "is_new": false}, {"model_id": "v2_h7_GLOBAL_LogisticRegression_N9", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 7, "n_features": 9, "F1_dir": 0.5675, "F1_UP_FORT": 0.3412, "F1_DOWN_FORT": 0.4212, "train_start": "2001-02-06", "sampler": "SMOTE", "best_params": "{}", "features": ["VIX_Price_zscore_60d__div__NFCI_vol_20d", "Core_CPI_zscore_60d", "VIX_Price_zscore_60d__div__CTAS_Cintas_vol_20d", "Nikkei_Japan_zscore_60d", "XLK_Tech_zscore_60d", "NOC_Northrop_ret_20d", "CLX_Clorox_vol_20d", "vix_zscore_10d__div__heston_var_ev_h7", "Nikkei_Japan_vol_20d"], "is_new": false}, {"model_id": "v2_h7_GLOBAL_LogisticRegression_N10", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 7, "n_features": 10, "F1_dir": 0.567, "F1_UP_FORT": 0.3338, "F1_DOWN_FORT": 0.4282, "train_start": "2001-02-06", "sampler": "SMOTE", "best_params": "{}", "features": ["VIX_Price_zscore_60d__div__NFCI_vol_20d", "Core_CPI_zscore_60d", "VIX_Price_zscore_60d__div__CTAS_Cintas_vol_20d", "Nikkei_Japan_zscore_60d", "XLK_Tech_zscore_60d", "NOC_Northrop_ret_20d", "CLX_Clorox_vol_20d", "vix_zscore_10d__div__heston_var_ev_h7", "Nikkei_Japan_vol_20d", "vix_vol_of_vol_10d__minus__NFCI_vol_20d"], "is_new": false}, {"model_id": "v2_h7_GLOBAL_LogisticRegression_N11", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 7, "n_features": 11, "F1_dir": 0.5642, "F1_UP_FORT": 0.3304, "F1_DOWN_FORT": 0.4185, "train_start": "2001-02-06", "sampler": "SMOTE", "best_params": "{}", "features": ["VIX_Price_zscore_60d__div__NFCI_vol_20d", "Core_CPI_zscore_60d", "VIX_Price_zscore_60d__div__CTAS_Cintas_vol_20d", "Nikkei_Japan_zscore_60d", "XLK_Tech_zscore_60d", "NOC_Northrop_ret_20d", "CLX_Clorox_vol_20d", "vix_zscore_10d__div__heston_var_ev_h7", "Nikkei_Japan_vol_20d", "vix_vol_of_vol_10d__minus__NFCI_vol_20d", "VRP__macross__hmm_p_stress"], "is_new": false}, {"model_id": "v2_h7_GLOBAL_LogisticRegression_N12", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 7, "n_features": 12, "F1_dir": 0.5606, "F1_UP_FORT": 0.3423, "F1_DOWN_FORT": 0.4332, "train_start": "2001-02-06", "sampler": "SMOTE", "best_params": "{}", "features": ["VIX_Price_zscore_60d__div__NFCI_vol_20d", "Core_CPI_zscore_60d", "VIX_Price_zscore_60d__div__CTAS_Cintas_vol_20d", "Nikkei_Japan_zscore_60d", "XLK_Tech_zscore_60d", "NOC_Northrop_ret_20d", "CLX_Clorox_vol_20d", "vix_zscore_10d__div__heston_var_ev_h7", "Nikkei_Japan_vol_20d", "vix_vol_of_vol_10d__minus__NFCI_vol_20d", "VRP__macross__hmm_p_stress", "HD_zscore_60d"], "is_new": false}, {"model_id": "v2_h7_GLOBAL_LogisticRegression_N13", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 7, "n_features": 13, "F1_dir": 0.5653, "F1_UP_FORT": 0.3404, "F1_DOWN_FORT": 0.4246, "train_start": "2001-02-06", "sampler": "SMOTE", "best_params": "{}", "features": ["VIX_Price_zscore_60d__div__NFCI_vol_20d", "Core_CPI_zscore_60d", "VIX_Price_zscore_60d__div__CTAS_Cintas_vol_20d", "Nikkei_Japan_zscore_60d", "XLK_Tech_zscore_60d", "NOC_Northrop_ret_20d", "CLX_Clorox_vol_20d", "vix_zscore_10d__div__heston_var_ev_h7", "Nikkei_Japan_vol_20d", "vix_vol_of_vol_10d__minus__NFCI_vol_20d", "VRP__macross__hmm_p_stress", "HD_zscore_60d", "Industrial_Production_zscore_60d"], "is_new": false}, {"model_id": "v2_h7_GLOBAL_LogisticRegression_N14", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 7, "n_features": 14, "F1_dir": 0.5945, "F1_UP_FORT": 0.3977, "F1_DOWN_FORT": 0.48, "train_start": "2001-02-06", "sampler": "SMOTE", "best_params": "{}", "features": ["VIX_Price_zscore_60d__div__NFCI_vol_20d", "Core_CPI_zscore_60d", "VIX_Price_zscore_60d__div__CTAS_Cintas_vol_20d", "Nikkei_Japan_zscore_60d", "XLK_Tech_zscore_60d", "NOC_Northrop_ret_20d", "CLX_Clorox_vol_20d", "vix_zscore_10d__div__heston_var_ev_h7", "Nikkei_Japan_vol_20d", "vix_vol_of_vol_10d__minus__NFCI_vol_20d", "VRP__macross__hmm_p_stress", "HD_zscore_60d", "Industrial_Production_zscore_60d", "NFCI_ret_5d__div__NFCI_vol_20d"], "is_new": false}, {"model_id": "v2_h7_GLOBAL_LogisticRegression_N15", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 7, "n_features": 15, "F1_dir": 0.5901, "F1_UP_FORT": 0.3865, "F1_DOWN_FORT": 0.4572, "train_start": "2001-02-06", "sampler": "SMOTE", "best_params": "{}", "features": ["VIX_Price_zscore_60d__div__NFCI_vol_20d", "Core_CPI_zscore_60d", "VIX_Price_zscore_60d__div__CTAS_Cintas_vol_20d", "Nikkei_Japan_zscore_60d", "XLK_Tech_zscore_60d", "NOC_Northrop_ret_20d", "CLX_Clorox_vol_20d", "vix_zscore_10d__div__heston_var_ev_h7", "Nikkei_Japan_vol_20d", "vix_vol_of_vol_10d__minus__NFCI_vol_20d", "VRP__macross__hmm_p_stress", "HD_zscore_60d", "Industrial_Production_zscore_60d", "NFCI_ret_5d__div__NFCI_vol_20d", "ASX_Australia_vol_20d"], "is_new": false}, {"model_id": "v2_h7_GLOBAL_LogisticRegression_N16", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 7, "n_features": 16, "F1_dir": 0.5873, "F1_UP_FORT": 0.3917, "F1_DOWN_FORT": 0.4525, "train_start": "2001-02-06", "sampler": "SMOTE", "best_params": "{}", "features": ["VIX_Price_zscore_60d__div__NFCI_vol_20d", "Core_CPI_zscore_60d", "VIX_Price_zscore_60d__div__CTAS_Cintas_vol_20d", "Nikkei_Japan_zscore_60d", "XLK_Tech_zscore_60d", "NOC_Northrop_ret_20d", "CLX_Clorox_vol_20d", "vix_zscore_10d__div__heston_var_ev_h7", "Nikkei_Japan_vol_20d", "vix_vol_of_vol_10d__minus__NFCI_vol_20d", "VRP__macross__hmm_p_stress", "HD_zscore_60d", "Industrial_Production_zscore_60d", "NFCI_ret_5d__div__NFCI_vol_20d", "ASX_Australia_vol_20d", "Michigan_Sentiment_ret_20d"], "is_new": false}, {"model_id": "v2_h7_GLOBAL_LogisticRegression_N17", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 7, "n_features": 17, "F1_dir": 0.5806, "F1_UP_FORT": 0.4086, "F1_DOWN_FORT": 0.4437, "train_start": "2001-02-06", "sampler": "SMOTE", "best_params": "{}", "features": ["VIX_Price_zscore_60d__div__NFCI_vol_20d", "Core_CPI_zscore_60d", "VIX_Price_zscore_60d__div__CTAS_Cintas_vol_20d", "Nikkei_Japan_zscore_60d", "XLK_Tech_zscore_60d", "NOC_Northrop_ret_20d", "CLX_Clorox_vol_20d", "vix_zscore_10d__div__heston_var_ev_h7", "Nikkei_Japan_vol_20d", "vix_vol_of_vol_10d__minus__NFCI_vol_20d", "VRP__macross__hmm_p_stress", "HD_zscore_60d", "Industrial_Production_zscore_60d", "NFCI_ret_5d__div__NFCI_vol_20d", "ASX_Australia_vol_20d", "Michigan_Sentiment_ret_20d", "XOM_ret_20d"], "is_new": false}, {"model_id": "v2_h7_GLOBAL_LogisticRegression_N18", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 7, "n_features": 18, "F1_dir": 0.5761, "F1_UP_FORT": 0.4038, "F1_DOWN_FORT": 0.4407, "train_start": "2001-02-06", "sampler": "SMOTE", "best_params": "{}", "features": ["VIX_Price_zscore_60d__div__NFCI_vol_20d", "Core_CPI_zscore_60d", "VIX_Price_zscore_60d__div__CTAS_Cintas_vol_20d", "Nikkei_Japan_zscore_60d", "XLK_Tech_zscore_60d", "NOC_Northrop_ret_20d", "CLX_Clorox_vol_20d", "vix_zscore_10d__div__heston_var_ev_h7", "Nikkei_Japan_vol_20d", "vix_vol_of_vol_10d__minus__NFCI_vol_20d", "VRP__macross__hmm_p_stress", "HD_zscore_60d", "Industrial_Production_zscore_60d", "NFCI_ret_5d__div__NFCI_vol_20d", "ASX_Australia_vol_20d", "Michigan_Sentiment_ret_20d", "XOM_ret_20d", "VIX_Price_zscore_60d__ret5x__NFCI_zscore_60d"], "is_new": false}, {"model_id": "v2_h7_GLOBAL_LogisticRegression_N19", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 7, "n_features": 19, "F1_dir": 0.5798, "F1_UP_FORT": 0.4043, "F1_DOWN_FORT": 0.4274, "train_start": "2001-02-06", "sampler": "SMOTE", "best_params": "{}", "features": ["VIX_Price_zscore_60d__div__NFCI_vol_20d", "Core_CPI_zscore_60d", "VIX_Price_zscore_60d__div__CTAS_Cintas_vol_20d", "Nikkei_Japan_zscore_60d", "XLK_Tech_zscore_60d", "NOC_Northrop_ret_20d", "CLX_Clorox_vol_20d", "vix_zscore_10d__div__heston_var_ev_h7", "Nikkei_Japan_vol_20d", "vix_vol_of_vol_10d__minus__NFCI_vol_20d", "VRP__macross__hmm_p_stress", "HD_zscore_60d", "Industrial_Production_zscore_60d", "NFCI_ret_5d__div__NFCI_vol_20d", "ASX_Australia_vol_20d", "Michigan_Sentiment_ret_20d", "XOM_ret_20d", "VIX_Price_zscore_60d__ret5x__NFCI_zscore_60d", "VIX_Price_zscore_60d__div__hmm_p_stress"], "is_new": false}, {"model_id": "v2_h7_GLOBAL_LogisticRegression_N20", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 7, "n_features": 20, "F1_dir": 0.5855, "F1_UP_FORT": 0.409, "F1_DOWN_FORT": 0.4324, "train_start": "2001-02-06", "sampler": "SMOTE", "best_params": "{}", "features": ["VIX_Price_zscore_60d__div__NFCI_vol_20d", "Core_CPI_zscore_60d", "VIX_Price_zscore_60d__div__CTAS_Cintas_vol_20d", "Nikkei_Japan_zscore_60d", "XLK_Tech_zscore_60d", "NOC_Northrop_ret_20d", "CLX_Clorox_vol_20d", "vix_zscore_10d__div__heston_var_ev_h7", "Nikkei_Japan_vol_20d", "vix_vol_of_vol_10d__minus__NFCI_vol_20d", "VRP__macross__hmm_p_stress", "HD_zscore_60d", "Industrial_Production_zscore_60d", "NFCI_ret_5d__div__NFCI_vol_20d", "ASX_Australia_vol_20d", "Michigan_Sentiment_ret_20d", "XOM_ret_20d", "VIX_Price_zscore_60d__ret5x__NFCI_zscore_60d", "VIX_Price_zscore_60d__div__hmm_p_stress", "VIX_Price_zscore_60d__prod__CTAS_Cintas_vol_20d"], "is_new": false}, {"model_id": "v2_h7_GLOBAL_LogisticRegression_N21", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 7, "n_features": 21, "F1_dir": 0.5999, "F1_UP_FORT": 0.3956, "F1_DOWN_FORT": 0.4593, "train_start": "2001-02-06", "sampler": "SMOTE", "best_params": "{}", "features": ["VIX_Price_zscore_60d__div__NFCI_vol_20d", "Core_CPI_zscore_60d", "VIX_Price_zscore_60d__div__CTAS_Cintas_vol_20d", "Nikkei_Japan_zscore_60d", "XLK_Tech_zscore_60d", "NOC_Northrop_ret_20d", "CLX_Clorox_vol_20d", "vix_zscore_10d__div__heston_var_ev_h7", "Nikkei_Japan_vol_20d", "vix_vol_of_vol_10d__minus__NFCI_vol_20d", "VRP__macross__hmm_p_stress", "HD_zscore_60d", "Industrial_Production_zscore_60d", "NFCI_ret_5d__div__NFCI_vol_20d", "ASX_Australia_vol_20d", "Michigan_Sentiment_ret_20d", "XOM_ret_20d", "VIX_Price_zscore_60d__ret5x__NFCI_zscore_60d", "VIX_Price_zscore_60d__div__hmm_p_stress", "VIX_Price_zscore_60d__prod__CTAS_Cintas_vol_20d", "vix_zscore_10d__div__vix_vol_of_vol_10d"], "is_new": false}, {"model_id": "v2_h7_GLOBAL_LogisticRegression_N22", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 7, "n_features": 22, "F1_dir": 0.5976, "F1_UP_FORT": 0.3898, "F1_DOWN_FORT": 0.4623, "train_start": "2001-02-06", "sampler": "SMOTE", "best_params": "{}", "features": ["VIX_Price_zscore_60d__div__NFCI_vol_20d", "Core_CPI_zscore_60d", "VIX_Price_zscore_60d__div__CTAS_Cintas_vol_20d", "Nikkei_Japan_zscore_60d", "XLK_Tech_zscore_60d", "NOC_Northrop_ret_20d", "CLX_Clorox_vol_20d", "vix_zscore_10d__div__heston_var_ev_h7", "Nikkei_Japan_vol_20d", "vix_vol_of_vol_10d__minus__NFCI_vol_20d", "VRP__macross__hmm_p_stress", "HD_zscore_60d", "Industrial_Production_zscore_60d", "NFCI_ret_5d__div__NFCI_vol_20d", "ASX_Australia_vol_20d", "Michigan_Sentiment_ret_20d", "XOM_ret_20d", "VIX_Price_zscore_60d__ret5x__NFCI_zscore_60d", "VIX_Price_zscore_60d__div__hmm_p_stress", "VIX_Price_zscore_60d__prod__CTAS_Cintas_vol_20d", "vix_zscore_10d__div__vix_vol_of_vol_10d", "VIX_Price_zscore_60d__zrel__SBUX_ret_5d"], "is_new": false}, {"model_id": "v2_h7_GLOBAL_LogisticRegression_N23", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 7, "n_features": 23, "F1_dir": 0.6013, "F1_UP_FORT": 0.3995, "F1_DOWN_FORT": 0.462, "train_start": "2001-02-06", "sampler": "SMOTE", "best_params": "{}", "features": ["VIX_Price_zscore_60d__div__NFCI_vol_20d", "Core_CPI_zscore_60d", "VIX_Price_zscore_60d__div__CTAS_Cintas_vol_20d", "Nikkei_Japan_zscore_60d", "XLK_Tech_zscore_60d", "NOC_Northrop_ret_20d", "CLX_Clorox_vol_20d", "vix_zscore_10d__div__heston_var_ev_h7", "Nikkei_Japan_vol_20d", "vix_vol_of_vol_10d__minus__NFCI_vol_20d", "VRP__macross__hmm_p_stress", "HD_zscore_60d", "Industrial_Production_zscore_60d", "NFCI_ret_5d__div__NFCI_vol_20d", "ASX_Australia_vol_20d", "Michigan_Sentiment_ret_20d", "XOM_ret_20d", "VIX_Price_zscore_60d__ret5x__NFCI_zscore_60d", "VIX_Price_zscore_60d__div__hmm_p_stress", "VIX_Price_zscore_60d__prod__CTAS_Cintas_vol_20d", "vix_zscore_10d__div__vix_vol_of_vol_10d", "VIX_Price_zscore_60d__zrel__SBUX_ret_5d", "VIX_Price_zscore_60d__minus__heston_xi"], "is_new": false}, {"model_id": "v2_h7_GLOBAL_RandomForest_Optuna_N23", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 7, "n_features": 23, "F1_dir": 0.6147, "F1_UP_FORT": 0.3838, "F1_DOWN_FORT": 0.506, "train_start": "2001-02-06", "sampler": "SMOTE", "best_params": "{}", "features": ["VIX_Price_zscore_60d__div__NFCI_vol_20d", "Core_CPI_zscore_60d", "VIX_Price_zscore_60d__div__CTAS_Cintas_vol_20d", "Nikkei_Japan_zscore_60d", "XLK_Tech_zscore_60d", "NOC_Northrop_ret_20d", "CLX_Clorox_vol_20d", "vix_zscore_10d__div__heston_var_ev_h7", "Nikkei_Japan_vol_20d", "vix_vol_of_vol_10d__minus__NFCI_vol_20d", "VRP__macross__hmm_p_stress", "HD_zscore_60d", "Industrial_Production_zscore_60d", "NFCI_ret_5d__div__NFCI_vol_20d", "ASX_Australia_vol_20d", "Michigan_Sentiment_ret_20d", "XOM_ret_20d", "VIX_Price_zscore_60d__ret5x__NFCI_zscore_60d", "VIX_Price_zscore_60d__div__hmm_p_stress", "VIX_Price_zscore_60d__prod__CTAS_Cintas_vol_20d", "vix_zscore_10d__div__vix_vol_of_vol_10d", "VIX_Price_zscore_60d__zrel__SBUX_ret_5d", "VIX_Price_zscore_60d__minus__heston_xi"], "is_new": false}, {"model_id": "v2_h7_GLOBAL_RandomForest_OptunaCal_N23", "algo": "RandomForestCal", "regime": "GLOBAL", "horizon": 7, "n_features": 23, "F1_dir": 0.6099, "F1_UP_FORT": 0.3891, "F1_DOWN_FORT": 0.5075, "train_start": "2001-02-06", "sampler": "SMOTE", "best_params": "{}", "features": ["VIX_Price_zscore_60d__div__NFCI_vol_20d", "Core_CPI_zscore_60d", "VIX_Price_zscore_60d__div__CTAS_Cintas_vol_20d", "Nikkei_Japan_zscore_60d", "XLK_Tech_zscore_60d", "NOC_Northrop_ret_20d", "CLX_Clorox_vol_20d", "vix_zscore_10d__div__heston_var_ev_h7", "Nikkei_Japan_vol_20d", "vix_vol_of_vol_10d__minus__NFCI_vol_20d", "VRP__macross__hmm_p_stress", "HD_zscore_60d", "Industrial_Production_zscore_60d", "NFCI_ret_5d__div__NFCI_vol_20d", "ASX_Australia_vol_20d", "Michigan_Sentiment_ret_20d", "XOM_ret_20d", "VIX_Price_zscore_60d__ret5x__NFCI_zscore_60d", "VIX_Price_zscore_60d__div__hmm_p_stress", "VIX_Price_zscore_60d__prod__CTAS_Cintas_vol_20d", "vix_zscore_10d__div__vix_vol_of_vol_10d", "VIX_Price_zscore_60d__zrel__SBUX_ret_5d", "VIX_Price_zscore_60d__minus__heston_xi"], "is_new": false}, {"model_id": "v1_h1_CALM_XGBoost_N5", "algo": "XGBoost", "regime": "CALM", "horizon": 1, "n_features": 5, "F1_dir": 0.5522, "F1_UP_FORT": 0.3043, "F1_DOWN_FORT": 0.34, "train_start": "2001-08-16", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["EWY_Korea_zscore_60d", "spx_vol_5d", "VZ_ret_5d__div__CAT_Caterpillar_ret_5d", "EQR_Equity_ret_1d", "BTI_BritishAmerican_ret_20d"], "is_new": false}, {"model_id": "v1_h1_CALM_XGBoost_N6", "algo": "XGBoost", "regime": "CALM", "horizon": 1, "n_features": 6, "F1_dir": 0.5318, "F1_UP_FORT": 0.2963, "F1_DOWN_FORT": 0.2737, "train_start": "2001-08-16", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["EWY_Korea_zscore_60d", "spx_vol_5d", "VZ_ret_5d__div__CAT_Caterpillar_ret_5d", "EQR_Equity_ret_1d", "BTI_BritishAmerican_ret_20d", "spx_drawdown_252d__zrel__WFC_WellsFargo_ret_1d"], "is_new": false}, {"model_id": "v1_h1_CALM_XGBoost_N7", "algo": "XGBoost", "regime": "CALM", "horizon": 1, "n_features": 7, "F1_dir": 0.557, "F1_UP_FORT": 0.2917, "F1_DOWN_FORT": 0.2857, "train_start": "2001-08-16", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["EWY_Korea_zscore_60d", "spx_vol_5d", "VZ_ret_5d__div__CAT_Caterpillar_ret_5d", "EQR_Equity_ret_1d", "BTI_BritishAmerican_ret_20d", "spx_drawdown_252d__zrel__WFC_WellsFargo_ret_1d", "WFC_WellsFargo_ret_1d__div__EWM_Malaysia_ret_1d"], "is_new": false}, {"model_id": "v1_h1_CALM_XGBoost_N8", "algo": "XGBoost", "regime": "CALM", "horizon": 1, "n_features": 8, "F1_dir": 0.5527, "F1_UP_FORT": 0.2366, "F1_DOWN_FORT": 0.2683, "train_start": "2001-08-16", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["EWY_Korea_zscore_60d", "spx_vol_5d", "VZ_ret_5d__div__CAT_Caterpillar_ret_5d", "EQR_Equity_ret_1d", "BTI_BritishAmerican_ret_20d", "spx_drawdown_252d__zrel__WFC_WellsFargo_ret_1d", "WFC_WellsFargo_ret_1d__div__EWM_Malaysia_ret_1d", "NFCI_ret_5d__div__COST_vol_20d"], "is_new": false}, {"model_id": "v1_h1_CALM_XGBoost_N9", "algo": "XGBoost", "regime": "CALM", "horizon": 1, "n_features": 9, "F1_dir": 0.5222, "F1_UP_FORT": 0.2718, "F1_DOWN_FORT": 0.2222, "train_start": "2001-08-16", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["EWY_Korea_zscore_60d", "spx_vol_5d", "VZ_ret_5d__div__CAT_Caterpillar_ret_5d", "EQR_Equity_ret_1d", "BTI_BritishAmerican_ret_20d", "spx_drawdown_252d__zrel__WFC_WellsFargo_ret_1d", "WFC_WellsFargo_ret_1d__div__EWM_Malaysia_ret_1d", "NFCI_ret_5d__div__COST_vol_20d", "3M_ret_1d__macross__CAT_Caterpillar_ret_5d"], "is_new": false}, {"model_id": "v1_h1_CALM_XGBoost_N10", "algo": "XGBoost", "regime": "CALM", "horizon": 1, "n_features": 10, "F1_dir": 0.5465, "F1_UP_FORT": 0.34, "F1_DOWN_FORT": 0.2439, "train_start": "2001-08-16", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["EWY_Korea_zscore_60d", "spx_vol_5d", "VZ_ret_5d__div__CAT_Caterpillar_ret_5d", "EQR_Equity_ret_1d", "BTI_BritishAmerican_ret_20d", "spx_drawdown_252d__zrel__WFC_WellsFargo_ret_1d", "WFC_WellsFargo_ret_1d__div__EWM_Malaysia_ret_1d", "NFCI_ret_5d__div__COST_vol_20d", "3M_ret_1d__macross__CAT_Caterpillar_ret_5d", "PFE_ret_1d"], "is_new": false}, {"model_id": "v1_h1_CALM_XGBoost_N11", "algo": "XGBoost", "regime": "CALM", "horizon": 1, "n_features": 11, "F1_dir": 0.5249, "F1_UP_FORT": 0.2828, "F1_DOWN_FORT": 0.2381, "train_start": "2001-08-16", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["EWY_Korea_zscore_60d", "spx_vol_5d", "VZ_ret_5d__div__CAT_Caterpillar_ret_5d", "EQR_Equity_ret_1d", "BTI_BritishAmerican_ret_20d", "spx_drawdown_252d__zrel__WFC_WellsFargo_ret_1d", "WFC_WellsFargo_ret_1d__div__EWM_Malaysia_ret_1d", "NFCI_ret_5d__div__COST_vol_20d", "3M_ret_1d__macross__CAT_Caterpillar_ret_5d", "PFE_ret_1d", "VLO_Valero_ret_1d__ret5x__EBAY_eBay_ret_20d"], "is_new": false}, {"model_id": "v1_h1_CALM_XGBoost_N12", "algo": "XGBoost", "regime": "CALM", "horizon": 1, "n_features": 12, "F1_dir": 0.5459, "F1_UP_FORT": 0.303, "F1_DOWN_FORT": 0.2308, "train_start": "2001-08-16", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["EWY_Korea_zscore_60d", "spx_vol_5d", "VZ_ret_5d__div__CAT_Caterpillar_ret_5d", "EQR_Equity_ret_1d", "BTI_BritishAmerican_ret_20d", "spx_drawdown_252d__zrel__WFC_WellsFargo_ret_1d", "WFC_WellsFargo_ret_1d__div__EWM_Malaysia_ret_1d", "NFCI_ret_5d__div__COST_vol_20d", "3M_ret_1d__macross__CAT_Caterpillar_ret_5d", "PFE_ret_1d", "VLO_Valero_ret_1d__ret5x__EBAY_eBay_ret_20d", "BBY_BestBuy_ret_1d__div__WFC_WellsFargo_ret_1d"], "is_new": false}, {"model_id": "v1_h1_CALM_XGBoost_N13", "algo": "XGBoost", "regime": "CALM", "horizon": 1, "n_features": 13, "F1_dir": 0.5485, "F1_UP_FORT": 0.3462, "F1_DOWN_FORT": 0.2093, "train_start": "2001-08-16", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["EWY_Korea_zscore_60d", "spx_vol_5d", "VZ_ret_5d__div__CAT_Caterpillar_ret_5d", "EQR_Equity_ret_1d", "BTI_BritishAmerican_ret_20d", "spx_drawdown_252d__zrel__WFC_WellsFargo_ret_1d", "WFC_WellsFargo_ret_1d__div__EWM_Malaysia_ret_1d", "NFCI_ret_5d__div__COST_vol_20d", "3M_ret_1d__macross__CAT_Caterpillar_ret_5d", "PFE_ret_1d", "VLO_Valero_ret_1d__ret5x__EBAY_eBay_ret_20d", "BBY_BestBuy_ret_1d__div__WFC_WellsFargo_ret_1d", "MS_MorganStanley_ret_1d"], "is_new": false}, {"model_id": "v1_h1_CALM_XGBoost_N14", "algo": "XGBoost", "regime": "CALM", "horizon": 1, "n_features": 14, "F1_dir": 0.5236, "F1_UP_FORT": 0.2772, "F1_DOWN_FORT": 0.1481, "train_start": "2001-08-16", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["EWY_Korea_zscore_60d", "spx_vol_5d", "VZ_ret_5d__div__CAT_Caterpillar_ret_5d", "EQR_Equity_ret_1d", "BTI_BritishAmerican_ret_20d", "spx_drawdown_252d__zrel__WFC_WellsFargo_ret_1d", "WFC_WellsFargo_ret_1d__div__EWM_Malaysia_ret_1d", "NFCI_ret_5d__div__COST_vol_20d", "3M_ret_1d__macross__CAT_Caterpillar_ret_5d", "PFE_ret_1d", "VLO_Valero_ret_1d__ret5x__EBAY_eBay_ret_20d", "BBY_BestBuy_ret_1d__div__WFC_WellsFargo_ret_1d", "MS_MorganStanley_ret_1d", "COST_vol_20d__zrel__WFC_WellsFargo_ret_1d"], "is_new": false}, {"model_id": "v1_h1_CALM_XGBoost_N15", "algo": "XGBoost", "regime": "CALM", "horizon": 1, "n_features": 15, "F1_dir": 0.5667, "F1_UP_FORT": 0.3366, "F1_DOWN_FORT": 0.2093, "train_start": "2001-08-16", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["EWY_Korea_zscore_60d", "spx_vol_5d", "VZ_ret_5d__div__CAT_Caterpillar_ret_5d", "EQR_Equity_ret_1d", "BTI_BritishAmerican_ret_20d", "spx_drawdown_252d__zrel__WFC_WellsFargo_ret_1d", "WFC_WellsFargo_ret_1d__div__EWM_Malaysia_ret_1d", "NFCI_ret_5d__div__COST_vol_20d", "3M_ret_1d__macross__CAT_Caterpillar_ret_5d", "PFE_ret_1d", "VLO_Valero_ret_1d__ret5x__EBAY_eBay_ret_20d", "BBY_BestBuy_ret_1d__div__WFC_WellsFargo_ret_1d", "MS_MorganStanley_ret_1d", "COST_vol_20d__zrel__WFC_WellsFargo_ret_1d", "GE_ret_1d"], "is_new": false}, {"model_id": "v1_h1_CALM_LightGBM_N5", "algo": "LightGBM", "regime": "CALM", "horizon": 1, "n_features": 5, "F1_dir": 0.5615, "F1_UP_FORT": 0.3696, "F1_DOWN_FORT": 0.2917, "train_start": "2001-08-16", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["EWY_Korea_zscore_60d", "spx_vol_5d", "VZ_ret_5d__div__CAT_Caterpillar_ret_5d", "EQR_Equity_ret_1d", "BTI_BritishAmerican_ret_20d"], "is_new": false}, {"model_id": "v1_h1_CALM_LightGBM_N6", "algo": "LightGBM", "regime": "CALM", "horizon": 1, "n_features": 6, "F1_dir": 0.5402, "F1_UP_FORT": 0.2469, "F1_DOWN_FORT": 0.3061, "train_start": "2001-08-16", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["EWY_Korea_zscore_60d", "spx_vol_5d", "VZ_ret_5d__div__CAT_Caterpillar_ret_5d", "EQR_Equity_ret_1d", "BTI_BritishAmerican_ret_20d", "spx_drawdown_252d__zrel__WFC_WellsFargo_ret_1d"], "is_new": false}, {"model_id": "v1_h1_CALM_LightGBM_N7", "algo": "LightGBM", "regime": "CALM", "horizon": 1, "n_features": 7, "F1_dir": 0.5593, "F1_UP_FORT": 0.2766, "F1_DOWN_FORT": 0.2247, "train_start": "2001-08-16", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["EWY_Korea_zscore_60d", "spx_vol_5d", "VZ_ret_5d__div__CAT_Caterpillar_ret_5d", "EQR_Equity_ret_1d", "BTI_BritishAmerican_ret_20d", "spx_drawdown_252d__zrel__WFC_WellsFargo_ret_1d", "WFC_WellsFargo_ret_1d__div__EWM_Malaysia_ret_1d"], "is_new": false}, {"model_id": "v1_h1_CALM_LightGBM_N8", "algo": "LightGBM", "regime": "CALM", "horizon": 1, "n_features": 8, "F1_dir": 0.5623, "F1_UP_FORT": 0.2105, "F1_DOWN_FORT": 0.241, "train_start": "2001-08-16", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["EWY_Korea_zscore_60d", "spx_vol_5d", "VZ_ret_5d__div__CAT_Caterpillar_ret_5d", "EQR_Equity_ret_1d", "BTI_BritishAmerican_ret_20d", "spx_drawdown_252d__zrel__WFC_WellsFargo_ret_1d", "WFC_WellsFargo_ret_1d__div__EWM_Malaysia_ret_1d", "NFCI_ret_5d__div__COST_vol_20d"], "is_new": false}, {"model_id": "v1_h1_CALM_LightGBM_N9", "algo": "LightGBM", "regime": "CALM", "horizon": 1, "n_features": 9, "F1_dir": 0.5683, "F1_UP_FORT": 0.2963, "F1_DOWN_FORT": 0.2927, "train_start": "2001-08-16", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["EWY_Korea_zscore_60d", "spx_vol_5d", "VZ_ret_5d__div__CAT_Caterpillar_ret_5d", "EQR_Equity_ret_1d", "BTI_BritishAmerican_ret_20d", "spx_drawdown_252d__zrel__WFC_WellsFargo_ret_1d", "WFC_WellsFargo_ret_1d__div__EWM_Malaysia_ret_1d", "NFCI_ret_5d__div__COST_vol_20d", "3M_ret_1d__macross__CAT_Caterpillar_ret_5d"], "is_new": false}, {"model_id": "v1_h1_CALM_LightGBM_N10", "algo": "LightGBM", "regime": "CALM", "horizon": 1, "n_features": 10, "F1_dir": 0.5815, "F1_UP_FORT": 0.25, "F1_DOWN_FORT": 0.3409, "train_start": "2001-08-16", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["EWY_Korea_zscore_60d", "spx_vol_5d", "VZ_ret_5d__div__CAT_Caterpillar_ret_5d", "EQR_Equity_ret_1d", "BTI_BritishAmerican_ret_20d", "spx_drawdown_252d__zrel__WFC_WellsFargo_ret_1d", "WFC_WellsFargo_ret_1d__div__EWM_Malaysia_ret_1d", "NFCI_ret_5d__div__COST_vol_20d", "3M_ret_1d__macross__CAT_Caterpillar_ret_5d", "PFE_ret_1d"], "is_new": false}, {"model_id": "v1_h1_CALM_LightGBM_N11", "algo": "LightGBM", "regime": "CALM", "horizon": 1, "n_features": 11, "F1_dir": 0.5823, "F1_UP_FORT": 0.3061, "F1_DOWN_FORT": 0.2381, "train_start": "2001-08-16", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["EWY_Korea_zscore_60d", "spx_vol_5d", "VZ_ret_5d__div__CAT_Caterpillar_ret_5d", "EQR_Equity_ret_1d", "BTI_BritishAmerican_ret_20d", "spx_drawdown_252d__zrel__WFC_WellsFargo_ret_1d", "WFC_WellsFargo_ret_1d__div__EWM_Malaysia_ret_1d", "NFCI_ret_5d__div__COST_vol_20d", "3M_ret_1d__macross__CAT_Caterpillar_ret_5d", "PFE_ret_1d", "VLO_Valero_ret_1d__ret5x__EBAY_eBay_ret_20d"], "is_new": false}, {"model_id": "v1_h1_CALM_LightGBM_N12", "algo": "LightGBM", "regime": "CALM", "horizon": 1, "n_features": 12, "F1_dir": 0.5911, "F1_UP_FORT": 0.3077, "F1_DOWN_FORT": 0.2326, "train_start": "2001-08-16", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["EWY_Korea_zscore_60d", "spx_vol_5d", "VZ_ret_5d__div__CAT_Caterpillar_ret_5d", "EQR_Equity_ret_1d", "BTI_BritishAmerican_ret_20d", "spx_drawdown_252d__zrel__WFC_WellsFargo_ret_1d", "WFC_WellsFargo_ret_1d__div__EWM_Malaysia_ret_1d", "NFCI_ret_5d__div__COST_vol_20d", "3M_ret_1d__macross__CAT_Caterpillar_ret_5d", "PFE_ret_1d", "VLO_Valero_ret_1d__ret5x__EBAY_eBay_ret_20d", "BBY_BestBuy_ret_1d__div__WFC_WellsFargo_ret_1d"], "is_new": false}, {"model_id": "v1_h1_CALM_LightGBM_N13", "algo": "LightGBM", "regime": "CALM", "horizon": 1, "n_features": 13, "F1_dir": 0.5701, "F1_UP_FORT": 0.3107, "F1_DOWN_FORT": 0.225, "train_start": "2001-08-16", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["EWY_Korea_zscore_60d", "spx_vol_5d", "VZ_ret_5d__div__CAT_Caterpillar_ret_5d", "EQR_Equity_ret_1d", "BTI_BritishAmerican_ret_20d", "spx_drawdown_252d__zrel__WFC_WellsFargo_ret_1d", "WFC_WellsFargo_ret_1d__div__EWM_Malaysia_ret_1d", "NFCI_ret_5d__div__COST_vol_20d", "3M_ret_1d__macross__CAT_Caterpillar_ret_5d", "PFE_ret_1d", "VLO_Valero_ret_1d__ret5x__EBAY_eBay_ret_20d", "BBY_BestBuy_ret_1d__div__WFC_WellsFargo_ret_1d", "MS_MorganStanley_ret_1d"], "is_new": false}, {"model_id": "v1_h1_CALM_LightGBM_N14", "algo": "LightGBM", "regime": "CALM", "horizon": 1, "n_features": 14, "F1_dir": 0.5375, "F1_UP_FORT": 0.3636, "F1_DOWN_FORT": 0.2169, "train_start": "2001-08-16", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["EWY_Korea_zscore_60d", "spx_vol_5d", "VZ_ret_5d__div__CAT_Caterpillar_ret_5d", "EQR_Equity_ret_1d", "BTI_BritishAmerican_ret_20d", "spx_drawdown_252d__zrel__WFC_WellsFargo_ret_1d", "WFC_WellsFargo_ret_1d__div__EWM_Malaysia_ret_1d", "NFCI_ret_5d__div__COST_vol_20d", "3M_ret_1d__macross__CAT_Caterpillar_ret_5d", "PFE_ret_1d", "VLO_Valero_ret_1d__ret5x__EBAY_eBay_ret_20d", "BBY_BestBuy_ret_1d__div__WFC_WellsFargo_ret_1d", "MS_MorganStanley_ret_1d", "COST_vol_20d__zrel__WFC_WellsFargo_ret_1d"], "is_new": false}, {"model_id": "v1_h1_CALM_LightGBM_N15", "algo": "LightGBM", "regime": "CALM", "horizon": 1, "n_features": 15, "F1_dir": 0.5579, "F1_UP_FORT": 0.3168, "F1_DOWN_FORT": 0.2759, "train_start": "2001-08-16", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["EWY_Korea_zscore_60d", "spx_vol_5d", "VZ_ret_5d__div__CAT_Caterpillar_ret_5d", "EQR_Equity_ret_1d", "BTI_BritishAmerican_ret_20d", "spx_drawdown_252d__zrel__WFC_WellsFargo_ret_1d", "WFC_WellsFargo_ret_1d__div__EWM_Malaysia_ret_1d", "NFCI_ret_5d__div__COST_vol_20d", "3M_ret_1d__macross__CAT_Caterpillar_ret_5d", "PFE_ret_1d", "VLO_Valero_ret_1d__ret5x__EBAY_eBay_ret_20d", "BBY_BestBuy_ret_1d__div__WFC_WellsFargo_ret_1d", "MS_MorganStanley_ret_1d", "COST_vol_20d__zrel__WFC_WellsFargo_ret_1d", "GE_ret_1d"], "is_new": false}, {"model_id": "v1_h1_CALM_GradientBoosting_N5", "algo": "GradientBoosting", "regime": "CALM", "horizon": 1, "n_features": 5, "F1_dir": 0.5285, "F1_UP_FORT": 0.241, "F1_DOWN_FORT": 0.3091, "train_start": "2001-08-16", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["EWY_Korea_zscore_60d", "spx_vol_5d", "VZ_ret_5d__div__CAT_Caterpillar_ret_5d", "EQR_Equity_ret_1d", "BTI_BritishAmerican_ret_20d"], "is_new": false}, {"model_id": "v1_h1_CALM_GradientBoosting_N6", "algo": "GradientBoosting", "regime": "CALM", "horizon": 1, "n_features": 6, "F1_dir": 0.5517, "F1_UP_FORT": 0.2667, "F1_DOWN_FORT": 0.2857, "train_start": "2001-08-16", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["EWY_Korea_zscore_60d", "spx_vol_5d", "VZ_ret_5d__div__CAT_Caterpillar_ret_5d", "EQR_Equity_ret_1d", "BTI_BritishAmerican_ret_20d", "spx_drawdown_252d__zrel__WFC_WellsFargo_ret_1d"], "is_new": false}, {"model_id": "v1_h1_CALM_GradientBoosting_N7", "algo": "GradientBoosting", "regime": "CALM", "horizon": 1, "n_features": 7, "F1_dir": 0.5535, "F1_UP_FORT": 0.2667, "F1_DOWN_FORT": 0.3371, "train_start": "2001-08-16", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["EWY_Korea_zscore_60d", "spx_vol_5d", "VZ_ret_5d__div__CAT_Caterpillar_ret_5d", "EQR_Equity_ret_1d", "BTI_BritishAmerican_ret_20d", "spx_drawdown_252d__zrel__WFC_WellsFargo_ret_1d", "WFC_WellsFargo_ret_1d__div__EWM_Malaysia_ret_1d"], "is_new": false}, {"model_id": "v1_h1_CALM_GradientBoosting_N8", "algo": "GradientBoosting", "regime": "CALM", "horizon": 1, "n_features": 8, "F1_dir": 0.5593, "F1_UP_FORT": 0.2581, "F1_DOWN_FORT": 0.3146, "train_start": "2001-08-16", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["EWY_Korea_zscore_60d", "spx_vol_5d", "VZ_ret_5d__div__CAT_Caterpillar_ret_5d", "EQR_Equity_ret_1d", "BTI_BritishAmerican_ret_20d", "spx_drawdown_252d__zrel__WFC_WellsFargo_ret_1d", "WFC_WellsFargo_ret_1d__div__EWM_Malaysia_ret_1d", "NFCI_ret_5d__div__COST_vol_20d"], "is_new": false}, {"model_id": "v1_h1_CALM_GradientBoosting_N9", "algo": "GradientBoosting", "regime": "CALM", "horizon": 1, "n_features": 9, "F1_dir": 0.5465, "F1_UP_FORT": 0.2692, "F1_DOWN_FORT": 0.2651, "train_start": "2001-08-16", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["EWY_Korea_zscore_60d", "spx_vol_5d", "VZ_ret_5d__div__CAT_Caterpillar_ret_5d", "EQR_Equity_ret_1d", "BTI_BritishAmerican_ret_20d", "spx_drawdown_252d__zrel__WFC_WellsFargo_ret_1d", "WFC_WellsFargo_ret_1d__div__EWM_Malaysia_ret_1d", "NFCI_ret_5d__div__COST_vol_20d", "3M_ret_1d__macross__CAT_Caterpillar_ret_5d"], "is_new": false}, {"model_id": "v1_h1_CALM_GradientBoosting_N10", "algo": "GradientBoosting", "regime": "CALM", "horizon": 1, "n_features": 10, "F1_dir": 0.5303, "F1_UP_FORT": 0.3, "F1_DOWN_FORT": 0.2651, "train_start": "2001-08-16", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["EWY_Korea_zscore_60d", "spx_vol_5d", "VZ_ret_5d__div__CAT_Caterpillar_ret_5d", "EQR_Equity_ret_1d", "BTI_BritishAmerican_ret_20d", "spx_drawdown_252d__zrel__WFC_WellsFargo_ret_1d", "WFC_WellsFargo_ret_1d__div__EWM_Malaysia_ret_1d", "NFCI_ret_5d__div__COST_vol_20d", "3M_ret_1d__macross__CAT_Caterpillar_ret_5d", "PFE_ret_1d"], "is_new": false}, {"model_id": "v1_h1_CALM_GradientBoosting_N11", "algo": "GradientBoosting", "regime": "CALM", "horizon": 1, "n_features": 11, "F1_dir": 0.5411, "F1_UP_FORT": 0.303, "F1_DOWN_FORT": 0.2716, "train_start": "2001-08-16", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["EWY_Korea_zscore_60d", "spx_vol_5d", "VZ_ret_5d__div__CAT_Caterpillar_ret_5d", "EQR_Equity_ret_1d", "BTI_BritishAmerican_ret_20d", "spx_drawdown_252d__zrel__WFC_WellsFargo_ret_1d", "WFC_WellsFargo_ret_1d__div__EWM_Malaysia_ret_1d", "NFCI_ret_5d__div__COST_vol_20d", "3M_ret_1d__macross__CAT_Caterpillar_ret_5d", "PFE_ret_1d", "VLO_Valero_ret_1d__ret5x__EBAY_eBay_ret_20d"], "is_new": false}, {"model_id": "v1_h1_CALM_GradientBoosting_N12", "algo": "GradientBoosting", "regime": "CALM", "horizon": 1, "n_features": 12, "F1_dir": 0.5497, "F1_UP_FORT": 0.2574, "F1_DOWN_FORT": 0.2532, "train_start": "2001-08-16", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["EWY_Korea_zscore_60d", "spx_vol_5d", "VZ_ret_5d__div__CAT_Caterpillar_ret_5d", "EQR_Equity_ret_1d", "BTI_BritishAmerican_ret_20d", "spx_drawdown_252d__zrel__WFC_WellsFargo_ret_1d", "WFC_WellsFargo_ret_1d__div__EWM_Malaysia_ret_1d", "NFCI_ret_5d__div__COST_vol_20d", "3M_ret_1d__macross__CAT_Caterpillar_ret_5d", "PFE_ret_1d", "VLO_Valero_ret_1d__ret5x__EBAY_eBay_ret_20d", "BBY_BestBuy_ret_1d__div__WFC_WellsFargo_ret_1d"], "is_new": false}, {"model_id": "v1_h1_CALM_GradientBoosting_N13", "algo": "GradientBoosting", "regime": "CALM", "horizon": 1, "n_features": 13, "F1_dir": 0.568, "F1_UP_FORT": 0.3364, "F1_DOWN_FORT": 0.2381, "train_start": "2001-08-16", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["EWY_Korea_zscore_60d", "spx_vol_5d", "VZ_ret_5d__div__CAT_Caterpillar_ret_5d", "EQR_Equity_ret_1d", "BTI_BritishAmerican_ret_20d", "spx_drawdown_252d__zrel__WFC_WellsFargo_ret_1d", "WFC_WellsFargo_ret_1d__div__EWM_Malaysia_ret_1d", "NFCI_ret_5d__div__COST_vol_20d", "3M_ret_1d__macross__CAT_Caterpillar_ret_5d", "PFE_ret_1d", "VLO_Valero_ret_1d__ret5x__EBAY_eBay_ret_20d", "BBY_BestBuy_ret_1d__div__WFC_WellsFargo_ret_1d", "MS_MorganStanley_ret_1d"], "is_new": false}, {"model_id": "v1_h1_CALM_GradientBoosting_N14", "algo": "GradientBoosting", "regime": "CALM", "horizon": 1, "n_features": 14, "F1_dir": 0.5648, "F1_UP_FORT": 0.3019, "F1_DOWN_FORT": 0.25, "train_start": "2001-08-16", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["EWY_Korea_zscore_60d", "spx_vol_5d", "VZ_ret_5d__div__CAT_Caterpillar_ret_5d", "EQR_Equity_ret_1d", "BTI_BritishAmerican_ret_20d", "spx_drawdown_252d__zrel__WFC_WellsFargo_ret_1d", "WFC_WellsFargo_ret_1d__div__EWM_Malaysia_ret_1d", "NFCI_ret_5d__div__COST_vol_20d", "3M_ret_1d__macross__CAT_Caterpillar_ret_5d", "PFE_ret_1d", "VLO_Valero_ret_1d__ret5x__EBAY_eBay_ret_20d", "BBY_BestBuy_ret_1d__div__WFC_WellsFargo_ret_1d", "MS_MorganStanley_ret_1d", "COST_vol_20d__zrel__WFC_WellsFargo_ret_1d"], "is_new": false}, {"model_id": "v1_h1_CALM_GradientBoosting_N15", "algo": "GradientBoosting", "regime": "CALM", "horizon": 1, "n_features": 15, "F1_dir": 0.5278, "F1_UP_FORT": 0.25, "F1_DOWN_FORT": 0.2069, "train_start": "2001-08-16", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["EWY_Korea_zscore_60d", "spx_vol_5d", "VZ_ret_5d__div__CAT_Caterpillar_ret_5d", "EQR_Equity_ret_1d", "BTI_BritishAmerican_ret_20d", "spx_drawdown_252d__zrel__WFC_WellsFargo_ret_1d", "WFC_WellsFargo_ret_1d__div__EWM_Malaysia_ret_1d", "NFCI_ret_5d__div__COST_vol_20d", "3M_ret_1d__macross__CAT_Caterpillar_ret_5d", "PFE_ret_1d", "VLO_Valero_ret_1d__ret5x__EBAY_eBay_ret_20d", "BBY_BestBuy_ret_1d__div__WFC_WellsFargo_ret_1d", "MS_MorganStanley_ret_1d", "COST_vol_20d__zrel__WFC_WellsFargo_ret_1d", "GE_ret_1d"], "is_new": false}, {"model_id": "v1_h1_CALM_RandomForest_N5", "algo": "RandomForest", "regime": "CALM", "horizon": 1, "n_features": 5, "F1_dir": 0.533, "F1_UP_FORT": 0.1951, "F1_DOWN_FORT": 0.354, "train_start": "2001-08-16", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["EWY_Korea_zscore_60d", "spx_vol_5d", "VZ_ret_5d__div__CAT_Caterpillar_ret_5d", "EQR_Equity_ret_1d", "BTI_BritishAmerican_ret_20d"], "is_new": false}, {"model_id": "v1_h1_CALM_RandomForest_N6", "algo": "RandomForest", "regime": "CALM", "horizon": 1, "n_features": 6, "F1_dir": 0.54, "F1_UP_FORT": 0.25, "F1_DOWN_FORT": 0.2772, "train_start": "2001-08-16", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["EWY_Korea_zscore_60d", "spx_vol_5d", "VZ_ret_5d__div__CAT_Caterpillar_ret_5d", "EQR_Equity_ret_1d", "BTI_BritishAmerican_ret_20d", "spx_drawdown_252d__zrel__WFC_WellsFargo_ret_1d"], "is_new": false}, {"model_id": "v1_h1_CALM_RandomForest_N7", "algo": "RandomForest", "regime": "CALM", "horizon": 1, "n_features": 7, "F1_dir": 0.5518, "F1_UP_FORT": 0.2449, "F1_DOWN_FORT": 0.1957, "train_start": "2001-08-16", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["EWY_Korea_zscore_60d", "spx_vol_5d", "VZ_ret_5d__div__CAT_Caterpillar_ret_5d", "EQR_Equity_ret_1d", "BTI_BritishAmerican_ret_20d", "spx_drawdown_252d__zrel__WFC_WellsFargo_ret_1d", "WFC_WellsFargo_ret_1d__div__EWM_Malaysia_ret_1d"], "is_new": false}, {"model_id": "v1_h1_CALM_RandomForest_N8", "algo": "RandomForest", "regime": "CALM", "horizon": 1, "n_features": 8, "F1_dir": 0.5667, "F1_UP_FORT": 0.2062, "F1_DOWN_FORT": 0.2921, "train_start": "2001-08-16", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["EWY_Korea_zscore_60d", "spx_vol_5d", "VZ_ret_5d__div__CAT_Caterpillar_ret_5d", "EQR_Equity_ret_1d", "BTI_BritishAmerican_ret_20d", "spx_drawdown_252d__zrel__WFC_WellsFargo_ret_1d", "WFC_WellsFargo_ret_1d__div__EWM_Malaysia_ret_1d", "NFCI_ret_5d__div__COST_vol_20d"], "is_new": false}, {"model_id": "v1_h1_CALM_RandomForest_N9", "algo": "RandomForest", "regime": "CALM", "horizon": 1, "n_features": 9, "F1_dir": 0.557, "F1_UP_FORT": 0.2245, "F1_DOWN_FORT": 0.1975, "train_start": "2001-08-16", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["EWY_Korea_zscore_60d", "spx_vol_5d", "VZ_ret_5d__div__CAT_Caterpillar_ret_5d", "EQR_Equity_ret_1d", "BTI_BritishAmerican_ret_20d", "spx_drawdown_252d__zrel__WFC_WellsFargo_ret_1d", "WFC_WellsFargo_ret_1d__div__EWM_Malaysia_ret_1d", "NFCI_ret_5d__div__COST_vol_20d", "3M_ret_1d__macross__CAT_Caterpillar_ret_5d"], "is_new": false}, {"model_id": "v1_h1_CALM_RandomForest_N10", "algo": "RandomForest", "regime": "CALM", "horizon": 1, "n_features": 10, "F1_dir": 0.5582, "F1_UP_FORT": 0.2449, "F1_DOWN_FORT": 0.2299, "train_start": "2001-08-16", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["EWY_Korea_zscore_60d", "spx_vol_5d", "VZ_ret_5d__div__CAT_Caterpillar_ret_5d", "EQR_Equity_ret_1d", "BTI_BritishAmerican_ret_20d", "spx_drawdown_252d__zrel__WFC_WellsFargo_ret_1d", "WFC_WellsFargo_ret_1d__div__EWM_Malaysia_ret_1d", "NFCI_ret_5d__div__COST_vol_20d", "3M_ret_1d__macross__CAT_Caterpillar_ret_5d", "PFE_ret_1d"], "is_new": false}, {"model_id": "v1_h1_CALM_RandomForest_N11", "algo": "RandomForest", "regime": "CALM", "horizon": 1, "n_features": 11, "F1_dir": 0.5508, "F1_UP_FORT": 0.2526, "F1_DOWN_FORT": 0.3023, "train_start": "2001-08-16", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["EWY_Korea_zscore_60d", "spx_vol_5d", "VZ_ret_5d__div__CAT_Caterpillar_ret_5d", "EQR_Equity_ret_1d", "BTI_BritishAmerican_ret_20d", "spx_drawdown_252d__zrel__WFC_WellsFargo_ret_1d", "WFC_WellsFargo_ret_1d__div__EWM_Malaysia_ret_1d", "NFCI_ret_5d__div__COST_vol_20d", "3M_ret_1d__macross__CAT_Caterpillar_ret_5d", "PFE_ret_1d", "VLO_Valero_ret_1d__ret5x__EBAY_eBay_ret_20d"], "is_new": false}, {"model_id": "v1_h1_CALM_RandomForest_N12", "algo": "RandomForest", "regime": "CALM", "horizon": 1, "n_features": 12, "F1_dir": 0.5459, "F1_UP_FORT": 0.2476, "F1_DOWN_FORT": 0.2532, "train_start": "2001-08-16", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["EWY_Korea_zscore_60d", "spx_vol_5d", "VZ_ret_5d__div__CAT_Caterpillar_ret_5d", "EQR_Equity_ret_1d", "BTI_BritishAmerican_ret_20d", "spx_drawdown_252d__zrel__WFC_WellsFargo_ret_1d", "WFC_WellsFargo_ret_1d__div__EWM_Malaysia_ret_1d", "NFCI_ret_5d__div__COST_vol_20d", "3M_ret_1d__macross__CAT_Caterpillar_ret_5d", "PFE_ret_1d", "VLO_Valero_ret_1d__ret5x__EBAY_eBay_ret_20d", "BBY_BestBuy_ret_1d__div__WFC_WellsFargo_ret_1d"], "is_new": false}, {"model_id": "v1_h1_CALM_RandomForest_N13", "algo": "RandomForest", "regime": "CALM", "horizon": 1, "n_features": 13, "F1_dir": 0.5788, "F1_UP_FORT": 0.1957, "F1_DOWN_FORT": 0.2391, "train_start": "2001-08-16", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["EWY_Korea_zscore_60d", "spx_vol_5d", "VZ_ret_5d__div__CAT_Caterpillar_ret_5d", "EQR_Equity_ret_1d", "BTI_BritishAmerican_ret_20d", "spx_drawdown_252d__zrel__WFC_WellsFargo_ret_1d", "WFC_WellsFargo_ret_1d__div__EWM_Malaysia_ret_1d", "NFCI_ret_5d__div__COST_vol_20d", "3M_ret_1d__macross__CAT_Caterpillar_ret_5d", "PFE_ret_1d", "VLO_Valero_ret_1d__ret5x__EBAY_eBay_ret_20d", "BBY_BestBuy_ret_1d__div__WFC_WellsFargo_ret_1d", "MS_MorganStanley_ret_1d"], "is_new": false}, {"model_id": "v1_h1_CALM_RandomForest_N14", "algo": "RandomForest", "regime": "CALM", "horizon": 1, "n_features": 14, "F1_dir": 0.5557, "F1_UP_FORT": 0.2708, "F1_DOWN_FORT": 0.2195, "train_start": "2001-08-16", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["EWY_Korea_zscore_60d", "spx_vol_5d", "VZ_ret_5d__div__CAT_Caterpillar_ret_5d", "EQR_Equity_ret_1d", "BTI_BritishAmerican_ret_20d", "spx_drawdown_252d__zrel__WFC_WellsFargo_ret_1d", "WFC_WellsFargo_ret_1d__div__EWM_Malaysia_ret_1d", "NFCI_ret_5d__div__COST_vol_20d", "3M_ret_1d__macross__CAT_Caterpillar_ret_5d", "PFE_ret_1d", "VLO_Valero_ret_1d__ret5x__EBAY_eBay_ret_20d", "BBY_BestBuy_ret_1d__div__WFC_WellsFargo_ret_1d", "MS_MorganStanley_ret_1d", "COST_vol_20d__zrel__WFC_WellsFargo_ret_1d"], "is_new": false}, {"model_id": "v1_h1_CALM_RandomForest_N15", "algo": "RandomForest", "regime": "CALM", "horizon": 1, "n_features": 15, "F1_dir": 0.5874, "F1_UP_FORT": 0.297, "F1_DOWN_FORT": 0.2169, "train_start": "2001-08-16", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["EWY_Korea_zscore_60d", "spx_vol_5d", "VZ_ret_5d__div__CAT_Caterpillar_ret_5d", "EQR_Equity_ret_1d", "BTI_BritishAmerican_ret_20d", "spx_drawdown_252d__zrel__WFC_WellsFargo_ret_1d", "WFC_WellsFargo_ret_1d__div__EWM_Malaysia_ret_1d", "NFCI_ret_5d__div__COST_vol_20d", "3M_ret_1d__macross__CAT_Caterpillar_ret_5d", "PFE_ret_1d", "VLO_Valero_ret_1d__ret5x__EBAY_eBay_ret_20d", "BBY_BestBuy_ret_1d__div__WFC_WellsFargo_ret_1d", "MS_MorganStanley_ret_1d", "COST_vol_20d__zrel__WFC_WellsFargo_ret_1d", "GE_ret_1d"], "is_new": false}, {"model_id": "v1_h1_CALM_LogisticRegression_N7", "algo": "LogisticRegression", "regime": "CALM", "horizon": 1, "n_features": 7, "F1_dir": 0.5272, "F1_UP_FORT": 0.2105, "F1_DOWN_FORT": 0.2796, "train_start": "2001-08-16", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["EWY_Korea_zscore_60d", "spx_vol_5d", "VZ_ret_5d__div__CAT_Caterpillar_ret_5d", "EQR_Equity_ret_1d", "BTI_BritishAmerican_ret_20d", "spx_drawdown_252d__zrel__WFC_WellsFargo_ret_1d", "WFC_WellsFargo_ret_1d__div__EWM_Malaysia_ret_1d"], "is_new": false}, {"model_id": "v1_h1_CALM_LogisticRegression_N8", "algo": "LogisticRegression", "regime": "CALM", "horizon": 1, "n_features": 8, "F1_dir": 0.5159, "F1_UP_FORT": 0.1905, "F1_DOWN_FORT": 0.2366, "train_start": "2001-08-16", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["EWY_Korea_zscore_60d", "spx_vol_5d", "VZ_ret_5d__div__CAT_Caterpillar_ret_5d", "EQR_Equity_ret_1d", "BTI_BritishAmerican_ret_20d", "spx_drawdown_252d__zrel__WFC_WellsFargo_ret_1d", "WFC_WellsFargo_ret_1d__div__EWM_Malaysia_ret_1d", "NFCI_ret_5d__div__COST_vol_20d"], "is_new": false}, {"model_id": "v1_h1_CALM_LogisticRegression_N9", "algo": "LogisticRegression", "regime": "CALM", "horizon": 1, "n_features": 9, "F1_dir": 0.5025, "F1_UP_FORT": 0.2247, "F1_DOWN_FORT": 0.2198, "train_start": "2001-08-16", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["EWY_Korea_zscore_60d", "spx_vol_5d", "VZ_ret_5d__div__CAT_Caterpillar_ret_5d", "EQR_Equity_ret_1d", "BTI_BritishAmerican_ret_20d", "spx_drawdown_252d__zrel__WFC_WellsFargo_ret_1d", "WFC_WellsFargo_ret_1d__div__EWM_Malaysia_ret_1d", "NFCI_ret_5d__div__COST_vol_20d", "3M_ret_1d__macross__CAT_Caterpillar_ret_5d"], "is_new": false}, {"model_id": "v1_h1_CALM_LogisticRegression_N12", "algo": "LogisticRegression", "regime": "CALM", "horizon": 1, "n_features": 12, "F1_dir": 0.5, "F1_UP_FORT": 0.2609, "F1_DOWN_FORT": 0.2474, "train_start": "2001-08-16", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["EWY_Korea_zscore_60d", "spx_vol_5d", "VZ_ret_5d__div__CAT_Caterpillar_ret_5d", "EQR_Equity_ret_1d", "BTI_BritishAmerican_ret_20d", "spx_drawdown_252d__zrel__WFC_WellsFargo_ret_1d", "WFC_WellsFargo_ret_1d__div__EWM_Malaysia_ret_1d", "NFCI_ret_5d__div__COST_vol_20d", "3M_ret_1d__macross__CAT_Caterpillar_ret_5d", "PFE_ret_1d", "VLO_Valero_ret_1d__ret5x__EBAY_eBay_ret_20d", "BBY_BestBuy_ret_1d__div__WFC_WellsFargo_ret_1d"], "is_new": false}, {"model_id": "v1_h1_CALM_LogisticRegression_N14", "algo": "LogisticRegression", "regime": "CALM", "horizon": 1, "n_features": 14, "F1_dir": 0.5185, "F1_UP_FORT": 0.3146, "F1_DOWN_FORT": 0.234, "train_start": "2001-08-16", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["EWY_Korea_zscore_60d", "spx_vol_5d", "VZ_ret_5d__div__CAT_Caterpillar_ret_5d", "EQR_Equity_ret_1d", "BTI_BritishAmerican_ret_20d", "spx_drawdown_252d__zrel__WFC_WellsFargo_ret_1d", "WFC_WellsFargo_ret_1d__div__EWM_Malaysia_ret_1d", "NFCI_ret_5d__div__COST_vol_20d", "3M_ret_1d__macross__CAT_Caterpillar_ret_5d", "PFE_ret_1d", "VLO_Valero_ret_1d__ret5x__EBAY_eBay_ret_20d", "BBY_BestBuy_ret_1d__div__WFC_WellsFargo_ret_1d", "MS_MorganStanley_ret_1d", "COST_vol_20d__zrel__WFC_WellsFargo_ret_1d"], "is_new": false}, {"model_id": "v1_h1_CALM_LogisticRegression_N15", "algo": "LogisticRegression", "regime": "CALM", "horizon": 1, "n_features": 15, "F1_dir": 0.5291, "F1_UP_FORT": 0.3333, "F1_DOWN_FORT": 0.2526, "train_start": "2001-08-16", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["EWY_Korea_zscore_60d", "spx_vol_5d", "VZ_ret_5d__div__CAT_Caterpillar_ret_5d", "EQR_Equity_ret_1d", "BTI_BritishAmerican_ret_20d", "spx_drawdown_252d__zrel__WFC_WellsFargo_ret_1d", "WFC_WellsFargo_ret_1d__div__EWM_Malaysia_ret_1d", "NFCI_ret_5d__div__COST_vol_20d", "3M_ret_1d__macross__CAT_Caterpillar_ret_5d", "PFE_ret_1d", "VLO_Valero_ret_1d__ret5x__EBAY_eBay_ret_20d", "BBY_BestBuy_ret_1d__div__WFC_WellsFargo_ret_1d", "MS_MorganStanley_ret_1d", "COST_vol_20d__zrel__WFC_WellsFargo_ret_1d", "GE_ret_1d"], "is_new": false}, {"model_id": "v1_h1_CALM_GradientBoosting_Optuna_N8", "algo": "GradientBoosting", "regime": "CALM", "horizon": 1, "n_features": 8, "F1_dir": 0.543, "F1_UP_FORT": 0.234, "F1_DOWN_FORT": 0.3023, "train_start": "2001-08-16", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["EWY_Korea_zscore_60d", "spx_vol_5d", "VZ_ret_5d__div__CAT_Caterpillar_ret_5d", "EQR_Equity_ret_1d", "BTI_BritishAmerican_ret_20d", "spx_drawdown_252d__zrel__WFC_WellsFargo_ret_1d", "WFC_WellsFargo_ret_1d__div__EWM_Malaysia_ret_1d", "NFCI_ret_5d__div__COST_vol_20d"], "is_new": false}, {"model_id": "v1_h1_CALM_GradientBoosting_OptunaCal_N8", "algo": "GradientBoostingCal", "regime": "CALM", "horizon": 1, "n_features": 8, "F1_dir": 0.5012, "F1_UP_FORT": 0.2022, "F1_DOWN_FORT": 0.2857, "train_start": "2001-08-16", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["EWY_Korea_zscore_60d", "spx_vol_5d", "VZ_ret_5d__div__CAT_Caterpillar_ret_5d", "EQR_Equity_ret_1d", "BTI_BritishAmerican_ret_20d", "spx_drawdown_252d__zrel__WFC_WellsFargo_ret_1d", "WFC_WellsFargo_ret_1d__div__EWM_Malaysia_ret_1d", "NFCI_ret_5d__div__COST_vol_20d"], "is_new": false}, {"model_id": "v1_h1_NORMAL_XGBoost_N5", "algo": "XGBoost", "regime": "NORMAL", "horizon": 1, "n_features": 5, "F1_dir": 0.5081, "F1_UP_FORT": 0.2577, "F1_DOWN_FORT": 0.3053, "train_start": "2000-11-16", "sampler": "SMOTETomek", "best_params": "{}", "features": ["heston_xi__minus__COF_CapitalOne_ret_5d", "XLY_Disc_zscore_60d__prod__ADM_ArcherDaniels_ret_1d", "US1Y_Rate_ret_20d", "heston_xi__minus__EQIX_Equinix_vol_20d", "VRP"], "is_new": false}, {"model_id": "v1_h1_NORMAL_XGBoost_N6", "algo": "XGBoost", "regime": "NORMAL", "horizon": 1, "n_features": 6, "F1_dir": 0.5083, "F1_UP_FORT": 0.3069, "F1_DOWN_FORT": 0.2879, "train_start": "2000-11-16", "sampler": "SMOTETomek", "best_params": "{}", "features": ["heston_xi__minus__COF_CapitalOne_ret_5d", "XLY_Disc_zscore_60d__prod__ADM_ArcherDaniels_ret_1d", "US1Y_Rate_ret_20d", "heston_xi__minus__EQIX_Equinix_vol_20d", "VRP", "heston_xi__div__ASML_ASML_vol_20d"], "is_new": false}, {"model_id": "v1_h1_NORMAL_XGBoost_N7", "algo": "XGBoost", "regime": "NORMAL", "horizon": 1, "n_features": 7, "F1_dir": 0.5249, "F1_UP_FORT": 0.272, "F1_DOWN_FORT": 0.2995, "train_start": "2000-11-16", "sampler": "SMOTETomek", "best_params": "{}", "features": ["heston_xi__minus__COF_CapitalOne_ret_5d", "XLY_Disc_zscore_60d__prod__ADM_ArcherDaniels_ret_1d", "US1Y_Rate_ret_20d", "heston_xi__minus__EQIX_Equinix_vol_20d", "VRP", "heston_xi__div__ASML_ASML_vol_20d", "vix_zscore_10d"], "is_new": false}, {"model_id": "v1_h1_NORMAL_XGBoost_N8", "algo": "XGBoost", "regime": "NORMAL", "horizon": 1, "n_features": 8, "F1_dir": 0.5229, "F1_UP_FORT": 0.2566, "F1_DOWN_FORT": 0.3636, "train_start": "2000-11-16", "sampler": "SMOTETomek", "best_params": "{}", "features": ["heston_xi__minus__COF_CapitalOne_ret_5d", "XLY_Disc_zscore_60d__prod__ADM_ArcherDaniels_ret_1d", "US1Y_Rate_ret_20d", "heston_xi__minus__EQIX_Equinix_vol_20d", "VRP", "heston_xi__div__ASML_ASML_vol_20d", "vix_zscore_10d", "IBEX_Spain_ret_20d__macross__ROST_RossStores_zscore_60d"], "is_new": false}, {"model_id": "v1_h1_NORMAL_XGBoost_N9", "algo": "XGBoost", "regime": "NORMAL", "horizon": 1, "n_features": 9, "F1_dir": 0.5205, "F1_UP_FORT": 0.2416, "F1_DOWN_FORT": 0.3096, "train_start": "2000-11-16", "sampler": "SMOTETomek", "best_params": "{}", "features": ["heston_xi__minus__COF_CapitalOne_ret_5d", "XLY_Disc_zscore_60d__prod__ADM_ArcherDaniels_ret_1d", "US1Y_Rate_ret_20d", "heston_xi__minus__EQIX_Equinix_vol_20d", "VRP", "heston_xi__div__ASML_ASML_vol_20d", "vix_zscore_10d", "IBEX_Spain_ret_20d__macross__ROST_RossStores_zscore_60d", "ROST_RossStores_ret_1d__ret5x__EQIX_Equinix_vol_20d"], "is_new": false}, {"model_id": "v1_h1_NORMAL_XGBoost_N10", "algo": "XGBoost", "regime": "NORMAL", "horizon": 1, "n_features": 10, "F1_dir": 0.5236, "F1_UP_FORT": 0.2689, "F1_DOWN_FORT": 0.3381, "train_start": "2000-11-16", "sampler": "SMOTETomek", "best_params": "{}", "features": ["heston_xi__minus__COF_CapitalOne_ret_5d", "XLY_Disc_zscore_60d__prod__ADM_ArcherDaniels_ret_1d", "US1Y_Rate_ret_20d", "heston_xi__minus__EQIX_Equinix_vol_20d", "VRP", "heston_xi__div__ASML_ASML_vol_20d", "vix_zscore_10d", "IBEX_Spain_ret_20d__macross__ROST_RossStores_zscore_60d", "ROST_RossStores_ret_1d__ret5x__EQIX_Equinix_vol_20d", "ROST_RossStores_ret_1d__macross__WMT_ret_20d"], "is_new": false}, {"model_id": "v1_h1_NORMAL_XGBoost_N11", "algo": "XGBoost", "regime": "NORMAL", "horizon": 1, "n_features": 11, "F1_dir": 0.5139, "F1_UP_FORT": 0.2191, "F1_DOWN_FORT": 0.3077, "train_start": "2000-11-16", "sampler": "SMOTETomek", "best_params": "{}", "features": ["heston_xi__minus__COF_CapitalOne_ret_5d", "XLY_Disc_zscore_60d__prod__ADM_ArcherDaniels_ret_1d", "US1Y_Rate_ret_20d", "heston_xi__minus__EQIX_Equinix_vol_20d", "VRP", "heston_xi__div__ASML_ASML_vol_20d", "vix_zscore_10d", "IBEX_Spain_ret_20d__macross__ROST_RossStores_zscore_60d", "ROST_RossStores_ret_1d__ret5x__EQIX_Equinix_vol_20d", "ROST_RossStores_ret_1d__macross__WMT_ret_20d", "ADM_ArcherDaniels_ret_1d__ret5x__ADBE_vol_20d"], "is_new": false}, {"model_id": "v1_h1_NORMAL_XGBoost_N12", "algo": "XGBoost", "regime": "NORMAL", "horizon": 1, "n_features": 12, "F1_dir": 0.5258, "F1_UP_FORT": 0.2701, "F1_DOWN_FORT": 0.335, "train_start": "2000-11-16", "sampler": "SMOTETomek", "best_params": "{}", "features": ["heston_xi__minus__COF_CapitalOne_ret_5d", "XLY_Disc_zscore_60d__prod__ADM_ArcherDaniels_ret_1d", "US1Y_Rate_ret_20d", "heston_xi__minus__EQIX_Equinix_vol_20d", "VRP", "heston_xi__div__ASML_ASML_vol_20d", "vix_zscore_10d", "IBEX_Spain_ret_20d__macross__ROST_RossStores_zscore_60d", "ROST_RossStores_ret_1d__ret5x__EQIX_Equinix_vol_20d", "ROST_RossStores_ret_1d__macross__WMT_ret_20d", "ADM_ArcherDaniels_ret_1d__ret5x__ADBE_vol_20d", "NFCI_ret_5d__minus__ORCL_ret_1d"], "is_new": false}, {"model_id": "v1_h1_NORMAL_XGBoost_N13", "algo": "XGBoost", "regime": "NORMAL", "horizon": 1, "n_features": 13, "F1_dir": 0.5371, "F1_UP_FORT": 0.2783, "F1_DOWN_FORT": 0.33, "train_start": "2000-11-16", "sampler": "SMOTETomek", "best_params": "{}", "features": ["heston_xi__minus__COF_CapitalOne_ret_5d", "XLY_Disc_zscore_60d__prod__ADM_ArcherDaniels_ret_1d", "US1Y_Rate_ret_20d", "heston_xi__minus__EQIX_Equinix_vol_20d", "VRP", "heston_xi__div__ASML_ASML_vol_20d", "vix_zscore_10d", "IBEX_Spain_ret_20d__macross__ROST_RossStores_zscore_60d", "ROST_RossStores_ret_1d__ret5x__EQIX_Equinix_vol_20d", "ROST_RossStores_ret_1d__macross__WMT_ret_20d", "ADM_ArcherDaniels_ret_1d__ret5x__ADBE_vol_20d", "NFCI_ret_5d__minus__ORCL_ret_1d", "MO_AltriaMG_ret_1d"], "is_new": false}, {"model_id": "v1_h1_NORMAL_XGBoost_N14", "algo": "XGBoost", "regime": "NORMAL", "horizon": 1, "n_features": 14, "F1_dir": 0.5379, "F1_UP_FORT": 0.2849, "F1_DOWN_FORT": 0.3409, "train_start": "2000-11-16", "sampler": "SMOTETomek", "best_params": "{}", "features": ["heston_xi__minus__COF_CapitalOne_ret_5d", "XLY_Disc_zscore_60d__prod__ADM_ArcherDaniels_ret_1d", "US1Y_Rate_ret_20d", "heston_xi__minus__EQIX_Equinix_vol_20d", "VRP", "heston_xi__div__ASML_ASML_vol_20d", "vix_zscore_10d", "IBEX_Spain_ret_20d__macross__ROST_RossStores_zscore_60d", "ROST_RossStores_ret_1d__ret5x__EQIX_Equinix_vol_20d", "ROST_RossStores_ret_1d__macross__WMT_ret_20d", "ADM_ArcherDaniels_ret_1d__ret5x__ADBE_vol_20d", "NFCI_ret_5d__minus__ORCL_ret_1d", "MO_AltriaMG_ret_1d", "NFCI_ret_5d__zrel__ORCL_ret_1d"], "is_new": false}, {"model_id": "v1_h1_NORMAL_XGBoost_N15", "algo": "XGBoost", "regime": "NORMAL", "horizon": 1, "n_features": 15, "F1_dir": 0.559, "F1_UP_FORT": 0.2831, "F1_DOWN_FORT": 0.3636, "train_start": "2000-11-16", "sampler": "SMOTETomek", "best_params": "{}", "features": ["heston_xi__minus__COF_CapitalOne_ret_5d", "XLY_Disc_zscore_60d__prod__ADM_ArcherDaniels_ret_1d", "US1Y_Rate_ret_20d", "heston_xi__minus__EQIX_Equinix_vol_20d", "VRP", "heston_xi__div__ASML_ASML_vol_20d", "vix_zscore_10d", "IBEX_Spain_ret_20d__macross__ROST_RossStores_zscore_60d", "ROST_RossStores_ret_1d__ret5x__EQIX_Equinix_vol_20d", "ROST_RossStores_ret_1d__macross__WMT_ret_20d", "ADM_ArcherDaniels_ret_1d__ret5x__ADBE_vol_20d", "NFCI_ret_5d__minus__ORCL_ret_1d", "MO_AltriaMG_ret_1d", "NFCI_ret_5d__zrel__ORCL_ret_1d", "ROST_RossStores_ret_1d__zrel__ORCL_ret_1d"], "is_new": false}, {"model_id": "v1_h1_NORMAL_XGBoost_N16", "algo": "XGBoost", "regime": "NORMAL", "horizon": 1, "n_features": 16, "F1_dir": 0.5489, "F1_UP_FORT": 0.2943, "F1_DOWN_FORT": 0.3484, "train_start": "2000-11-16", "sampler": "SMOTETomek", "best_params": "{}", "features": ["heston_xi__minus__COF_CapitalOne_ret_5d", "XLY_Disc_zscore_60d__prod__ADM_ArcherDaniels_ret_1d", "US1Y_Rate_ret_20d", "heston_xi__minus__EQIX_Equinix_vol_20d", "VRP", "heston_xi__div__ASML_ASML_vol_20d", "vix_zscore_10d", "IBEX_Spain_ret_20d__macross__ROST_RossStores_zscore_60d", "ROST_RossStores_ret_1d__ret5x__EQIX_Equinix_vol_20d", "ROST_RossStores_ret_1d__macross__WMT_ret_20d", "ADM_ArcherDaniels_ret_1d__ret5x__ADBE_vol_20d", "NFCI_ret_5d__minus__ORCL_ret_1d", "MO_AltriaMG_ret_1d", "NFCI_ret_5d__zrel__ORCL_ret_1d", "ROST_RossStores_ret_1d__zrel__ORCL_ret_1d", "heston_xi__minus__ADBE_vol_20d"], "is_new": false}, {"model_id": "v1_h1_NORMAL_XGBoost_N17", "algo": "XGBoost", "regime": "NORMAL", "horizon": 1, "n_features": 17, "F1_dir": 0.5289, "F1_UP_FORT": 0.2493, "F1_DOWN_FORT": 0.3457, "train_start": "2000-11-16", "sampler": "SMOTETomek", "best_params": "{}", "features": ["heston_xi__minus__COF_CapitalOne_ret_5d", "XLY_Disc_zscore_60d__prod__ADM_ArcherDaniels_ret_1d", "US1Y_Rate_ret_20d", "heston_xi__minus__EQIX_Equinix_vol_20d", "VRP", "heston_xi__div__ASML_ASML_vol_20d", "vix_zscore_10d", "IBEX_Spain_ret_20d__macross__ROST_RossStores_zscore_60d", "ROST_RossStores_ret_1d__ret5x__EQIX_Equinix_vol_20d", "ROST_RossStores_ret_1d__macross__WMT_ret_20d", "ADM_ArcherDaniels_ret_1d__ret5x__ADBE_vol_20d", "NFCI_ret_5d__minus__ORCL_ret_1d", "MO_AltriaMG_ret_1d", "NFCI_ret_5d__zrel__ORCL_ret_1d", "ROST_RossStores_ret_1d__zrel__ORCL_ret_1d", "heston_xi__minus__ADBE_vol_20d", "IBEX_Spain_ret_20d__div__ROST_RossStores_zscore_60d"], "is_new": false}, {"model_id": "v1_h1_NORMAL_XGBoost_N18", "algo": "XGBoost", "regime": "NORMAL", "horizon": 1, "n_features": 18, "F1_dir": 0.5423, "F1_UP_FORT": 0.2454, "F1_DOWN_FORT": 0.3437, "train_start": "2000-11-16", "sampler": "SMOTETomek", "best_params": "{}", "features": ["heston_xi__minus__COF_CapitalOne_ret_5d", "XLY_Disc_zscore_60d__prod__ADM_ArcherDaniels_ret_1d", "US1Y_Rate_ret_20d", "heston_xi__minus__EQIX_Equinix_vol_20d", "VRP", "heston_xi__div__ASML_ASML_vol_20d", "vix_zscore_10d", "IBEX_Spain_ret_20d__macross__ROST_RossStores_zscore_60d", "ROST_RossStores_ret_1d__ret5x__EQIX_Equinix_vol_20d", "ROST_RossStores_ret_1d__macross__WMT_ret_20d", "ADM_ArcherDaniels_ret_1d__ret5x__ADBE_vol_20d", "NFCI_ret_5d__minus__ORCL_ret_1d", "MO_AltriaMG_ret_1d", "NFCI_ret_5d__zrel__ORCL_ret_1d", "ROST_RossStores_ret_1d__zrel__ORCL_ret_1d", "heston_xi__minus__ADBE_vol_20d", "IBEX_Spain_ret_20d__div__ROST_RossStores_zscore_60d", "ES_Evergy_ret_1d"], "is_new": false}, {"model_id": "v1_h1_NORMAL_XGBoost_N19", "algo": "XGBoost", "regime": "NORMAL", "horizon": 1, "n_features": 19, "F1_dir": 0.5259, "F1_UP_FORT": 0.3226, "F1_DOWN_FORT": 0.3112, "train_start": "2000-11-16", "sampler": "SMOTETomek", "best_params": "{}", "features": ["heston_xi__minus__COF_CapitalOne_ret_5d", "XLY_Disc_zscore_60d__prod__ADM_ArcherDaniels_ret_1d", "US1Y_Rate_ret_20d", "heston_xi__minus__EQIX_Equinix_vol_20d", "VRP", "heston_xi__div__ASML_ASML_vol_20d", "vix_zscore_10d", "IBEX_Spain_ret_20d__macross__ROST_RossStores_zscore_60d", "ROST_RossStores_ret_1d__ret5x__EQIX_Equinix_vol_20d", "ROST_RossStores_ret_1d__macross__WMT_ret_20d", "ADM_ArcherDaniels_ret_1d__ret5x__ADBE_vol_20d", "NFCI_ret_5d__minus__ORCL_ret_1d", "MO_AltriaMG_ret_1d", "NFCI_ret_5d__zrel__ORCL_ret_1d", "ROST_RossStores_ret_1d__zrel__ORCL_ret_1d", "heston_xi__minus__ADBE_vol_20d", "IBEX_Spain_ret_20d__div__ROST_RossStores_zscore_60d", "ES_Evergy_ret_1d", "MSTR_Bitcoin3_ret_1d"], "is_new": false}, {"model_id": "v1_h1_NORMAL_XGBoost_N20", "algo": "XGBoost", "regime": "NORMAL", "horizon": 1, "n_features": 20, "F1_dir": 0.5439, "F1_UP_FORT": 0.3003, "F1_DOWN_FORT": 0.3309, "train_start": "2000-11-16", "sampler": "SMOTETomek", "best_params": "{}", "features": ["heston_xi__minus__COF_CapitalOne_ret_5d", "XLY_Disc_zscore_60d__prod__ADM_ArcherDaniels_ret_1d", "US1Y_Rate_ret_20d", "heston_xi__minus__EQIX_Equinix_vol_20d", "VRP", "heston_xi__div__ASML_ASML_vol_20d", "vix_zscore_10d", "IBEX_Spain_ret_20d__macross__ROST_RossStores_zscore_60d", "ROST_RossStores_ret_1d__ret5x__EQIX_Equinix_vol_20d", "ROST_RossStores_ret_1d__macross__WMT_ret_20d", "ADM_ArcherDaniels_ret_1d__ret5x__ADBE_vol_20d", "NFCI_ret_5d__minus__ORCL_ret_1d", "MO_AltriaMG_ret_1d", "NFCI_ret_5d__zrel__ORCL_ret_1d", "ROST_RossStores_ret_1d__zrel__ORCL_ret_1d", "heston_xi__minus__ADBE_vol_20d", "IBEX_Spain_ret_20d__div__ROST_RossStores_zscore_60d", "ES_Evergy_ret_1d", "MSTR_Bitcoin3_ret_1d", "LMT_LockheedMartin_ret_1d"], "is_new": false}, {"model_id": "v1_h1_NORMAL_XGBoost_N21", "algo": "XGBoost", "regime": "NORMAL", "horizon": 1, "n_features": 21, "F1_dir": 0.5301, "F1_UP_FORT": 0.3081, "F1_DOWN_FORT": 0.3056, "train_start": "2000-11-16", "sampler": "SMOTETomek", "best_params": "{}", "features": ["heston_xi__minus__COF_CapitalOne_ret_5d", "XLY_Disc_zscore_60d__prod__ADM_ArcherDaniels_ret_1d", "US1Y_Rate_ret_20d", "heston_xi__minus__EQIX_Equinix_vol_20d", "VRP", "heston_xi__div__ASML_ASML_vol_20d", "vix_zscore_10d", "IBEX_Spain_ret_20d__macross__ROST_RossStores_zscore_60d", "ROST_RossStores_ret_1d__ret5x__EQIX_Equinix_vol_20d", "ROST_RossStores_ret_1d__macross__WMT_ret_20d", "ADM_ArcherDaniels_ret_1d__ret5x__ADBE_vol_20d", "NFCI_ret_5d__minus__ORCL_ret_1d", "MO_AltriaMG_ret_1d", "NFCI_ret_5d__zrel__ORCL_ret_1d", "ROST_RossStores_ret_1d__zrel__ORCL_ret_1d", "heston_xi__minus__ADBE_vol_20d", "IBEX_Spain_ret_20d__div__ROST_RossStores_zscore_60d", "ES_Evergy_ret_1d", "MSTR_Bitcoin3_ret_1d", "LMT_LockheedMartin_ret_1d", "INTC_ret_1d"], "is_new": false}, {"model_id": "v1_h1_NORMAL_XGBoost_N22", "algo": "XGBoost", "regime": "NORMAL", "horizon": 1, "n_features": 22, "F1_dir": 0.5234, "F1_UP_FORT": 0.2914, "F1_DOWN_FORT": 0.3245, "train_start": "2000-11-16", "sampler": "SMOTETomek", "best_params": "{}", "features": ["heston_xi__minus__COF_CapitalOne_ret_5d", "XLY_Disc_zscore_60d__prod__ADM_ArcherDaniels_ret_1d", "US1Y_Rate_ret_20d", "heston_xi__minus__EQIX_Equinix_vol_20d", "VRP", "heston_xi__div__ASML_ASML_vol_20d", "vix_zscore_10d", "IBEX_Spain_ret_20d__macross__ROST_RossStores_zscore_60d", "ROST_RossStores_ret_1d__ret5x__EQIX_Equinix_vol_20d", "ROST_RossStores_ret_1d__macross__WMT_ret_20d", "ADM_ArcherDaniels_ret_1d__ret5x__ADBE_vol_20d", "NFCI_ret_5d__minus__ORCL_ret_1d", "MO_AltriaMG_ret_1d", "NFCI_ret_5d__zrel__ORCL_ret_1d", "ROST_RossStores_ret_1d__zrel__ORCL_ret_1d", "heston_xi__minus__ADBE_vol_20d", "IBEX_Spain_ret_20d__div__ROST_RossStores_zscore_60d", "ES_Evergy_ret_1d", "MSTR_Bitcoin3_ret_1d", "LMT_LockheedMartin_ret_1d", "INTC_ret_1d", "VIX_Price_zscore_60d__minus__ROST_RossStores_zscore_60d"], "is_new": false}, {"model_id": "v1_h1_NORMAL_XGBoost_N23", "algo": "XGBoost", "regime": "NORMAL", "horizon": 1, "n_features": 23, "F1_dir": 0.5358, "F1_UP_FORT": 0.3112, "F1_DOWN_FORT": 0.3351, "train_start": "2000-11-16", "sampler": "SMOTETomek", "best_params": "{}", "features": ["heston_xi__minus__COF_CapitalOne_ret_5d", "XLY_Disc_zscore_60d__prod__ADM_ArcherDaniels_ret_1d", "US1Y_Rate_ret_20d", "heston_xi__minus__EQIX_Equinix_vol_20d", "VRP", "heston_xi__div__ASML_ASML_vol_20d", "vix_zscore_10d", "IBEX_Spain_ret_20d__macross__ROST_RossStores_zscore_60d", "ROST_RossStores_ret_1d__ret5x__EQIX_Equinix_vol_20d", "ROST_RossStores_ret_1d__macross__WMT_ret_20d", "ADM_ArcherDaniels_ret_1d__ret5x__ADBE_vol_20d", "NFCI_ret_5d__minus__ORCL_ret_1d", "MO_AltriaMG_ret_1d", "NFCI_ret_5d__zrel__ORCL_ret_1d", "ROST_RossStores_ret_1d__zrel__ORCL_ret_1d", "heston_xi__minus__ADBE_vol_20d", "IBEX_Spain_ret_20d__div__ROST_RossStores_zscore_60d", "ES_Evergy_ret_1d", "MSTR_Bitcoin3_ret_1d", "LMT_LockheedMartin_ret_1d", "INTC_ret_1d", "VIX_Price_zscore_60d__minus__ROST_RossStores_zscore_60d", "VIX_Price_zscore_60d__prod__WMT_ret_20d"], "is_new": false}, {"model_id": "v1_h1_NORMAL_XGBoost_N24", "algo": "XGBoost", "regime": "NORMAL", "horizon": 1, "n_features": 24, "F1_dir": 0.5297, "F1_UP_FORT": 0.3152, "F1_DOWN_FORT": 0.3276, "train_start": "2000-11-16", "sampler": "SMOTETomek", "best_params": "{}", "features": ["heston_xi__minus__COF_CapitalOne_ret_5d", "XLY_Disc_zscore_60d__prod__ADM_ArcherDaniels_ret_1d", "US1Y_Rate_ret_20d", "heston_xi__minus__EQIX_Equinix_vol_20d", "VRP", "heston_xi__div__ASML_ASML_vol_20d", "vix_zscore_10d", "IBEX_Spain_ret_20d__macross__ROST_RossStores_zscore_60d", "ROST_RossStores_ret_1d__ret5x__EQIX_Equinix_vol_20d", "ROST_RossStores_ret_1d__macross__WMT_ret_20d", "ADM_ArcherDaniels_ret_1d__ret5x__ADBE_vol_20d", "NFCI_ret_5d__minus__ORCL_ret_1d", "MO_AltriaMG_ret_1d", "NFCI_ret_5d__zrel__ORCL_ret_1d", "ROST_RossStores_ret_1d__zrel__ORCL_ret_1d", "heston_xi__minus__ADBE_vol_20d", "IBEX_Spain_ret_20d__div__ROST_RossStores_zscore_60d", "ES_Evergy_ret_1d", "MSTR_Bitcoin3_ret_1d", "LMT_LockheedMartin_ret_1d", "INTC_ret_1d", "VIX_Price_zscore_60d__minus__ROST_RossStores_zscore_60d", "VIX_Price_zscore_60d__prod__WMT_ret_20d", "heston_xi__zrel__XLY_Disc_zscore_60d"], "is_new": false}, {"model_id": "v1_h1_NORMAL_XGBoost_N25", "algo": "XGBoost", "regime": "NORMAL", "horizon": 1, "n_features": 25, "F1_dir": 0.5005, "F1_UP_FORT": 0.2523, "F1_DOWN_FORT": 0.3184, "train_start": "2000-11-16", "sampler": "SMOTETomek", "best_params": "{}", "features": ["heston_xi__minus__COF_CapitalOne_ret_5d", "XLY_Disc_zscore_60d__prod__ADM_ArcherDaniels_ret_1d", "US1Y_Rate_ret_20d", "heston_xi__minus__EQIX_Equinix_vol_20d", "VRP", "heston_xi__div__ASML_ASML_vol_20d", "vix_zscore_10d", "IBEX_Spain_ret_20d__macross__ROST_RossStores_zscore_60d", "ROST_RossStores_ret_1d__ret5x__EQIX_Equinix_vol_20d", "ROST_RossStores_ret_1d__macross__WMT_ret_20d", "ADM_ArcherDaniels_ret_1d__ret5x__ADBE_vol_20d", "NFCI_ret_5d__minus__ORCL_ret_1d", "MO_AltriaMG_ret_1d", "NFCI_ret_5d__zrel__ORCL_ret_1d", "ROST_RossStores_ret_1d__zrel__ORCL_ret_1d", "heston_xi__minus__ADBE_vol_20d", "IBEX_Spain_ret_20d__div__ROST_RossStores_zscore_60d", "ES_Evergy_ret_1d", "MSTR_Bitcoin3_ret_1d", "LMT_LockheedMartin_ret_1d", "INTC_ret_1d", "VIX_Price_zscore_60d__minus__ROST_RossStores_zscore_60d", "VIX_Price_zscore_60d__prod__WMT_ret_20d", "heston_xi__zrel__XLY_Disc_zscore_60d", "YUM_YumBrands_ret_1d__minus__WMT_ret_20d"], "is_new": false}, {"model_id": "v1_h1_NORMAL_LightGBM_N5", "algo": "LightGBM", "regime": "NORMAL", "horizon": 1, "n_features": 5, "F1_dir": 0.5108, "F1_UP_FORT": 0.2698, "F1_DOWN_FORT": 0.3206, "train_start": "2000-11-16", "sampler": "SMOTETomek", "best_params": "{}", "features": ["heston_xi__minus__COF_CapitalOne_ret_5d", "XLY_Disc_zscore_60d__prod__ADM_ArcherDaniels_ret_1d", "US1Y_Rate_ret_20d", "heston_xi__minus__EQIX_Equinix_vol_20d", "VRP"], "is_new": false}, {"model_id": "v1_h1_NORMAL_LightGBM_N7", "algo": "LightGBM", "regime": "NORMAL", "horizon": 1, "n_features": 7, "F1_dir": 0.5144, "F1_UP_FORT": 0.2577, "F1_DOWN_FORT": 0.2984, "train_start": "2000-11-16", "sampler": "SMOTETomek", "best_params": "{}", "features": ["heston_xi__minus__COF_CapitalOne_ret_5d", "XLY_Disc_zscore_60d__prod__ADM_ArcherDaniels_ret_1d", "US1Y_Rate_ret_20d", "heston_xi__minus__EQIX_Equinix_vol_20d", "VRP", "heston_xi__div__ASML_ASML_vol_20d", "vix_zscore_10d"], "is_new": false}, {"model_id": "v1_h1_NORMAL_LightGBM_N8", "algo": "LightGBM", "regime": "NORMAL", "horizon": 1, "n_features": 8, "F1_dir": 0.5511, "F1_UP_FORT": 0.2557, "F1_DOWN_FORT": 0.2969, "train_start": "2000-11-16", "sampler": "SMOTETomek", "best_params": "{}", "features": ["heston_xi__minus__COF_CapitalOne_ret_5d", "XLY_Disc_zscore_60d__prod__ADM_ArcherDaniels_ret_1d", "US1Y_Rate_ret_20d", "heston_xi__minus__EQIX_Equinix_vol_20d", "VRP", "heston_xi__div__ASML_ASML_vol_20d", "vix_zscore_10d", "IBEX_Spain_ret_20d__macross__ROST_RossStores_zscore_60d"], "is_new": false}, {"model_id": "v1_h1_NORMAL_LightGBM_N9", "algo": "LightGBM", "regime": "NORMAL", "horizon": 1, "n_features": 9, "F1_dir": 0.5253, "F1_UP_FORT": 0.2294, "F1_DOWN_FORT": 0.3008, "train_start": "2000-11-16", "sampler": "SMOTETomek", "best_params": "{}", "features": ["heston_xi__minus__COF_CapitalOne_ret_5d", "XLY_Disc_zscore_60d__prod__ADM_ArcherDaniels_ret_1d", "US1Y_Rate_ret_20d", "heston_xi__minus__EQIX_Equinix_vol_20d", "VRP", "heston_xi__div__ASML_ASML_vol_20d", "vix_zscore_10d", "IBEX_Spain_ret_20d__macross__ROST_RossStores_zscore_60d", "ROST_RossStores_ret_1d__ret5x__EQIX_Equinix_vol_20d"], "is_new": false}, {"model_id": "v1_h1_NORMAL_LightGBM_N10", "algo": "LightGBM", "regime": "NORMAL", "horizon": 1, "n_features": 10, "F1_dir": 0.5537, "F1_UP_FORT": 0.2248, "F1_DOWN_FORT": 0.3053, "train_start": "2000-11-16", "sampler": "SMOTETomek", "best_params": "{}", "features": ["heston_xi__minus__COF_CapitalOne_ret_5d", "XLY_Disc_zscore_60d__prod__ADM_ArcherDaniels_ret_1d", "US1Y_Rate_ret_20d", "heston_xi__minus__EQIX_Equinix_vol_20d", "VRP", "heston_xi__div__ASML_ASML_vol_20d", "vix_zscore_10d", "IBEX_Spain_ret_20d__macross__ROST_RossStores_zscore_60d", "ROST_RossStores_ret_1d__ret5x__EQIX_Equinix_vol_20d", "ROST_RossStores_ret_1d__macross__WMT_ret_20d"], "is_new": false}, {"model_id": "v1_h1_NORMAL_LightGBM_N11", "algo": "LightGBM", "regime": "NORMAL", "horizon": 1, "n_features": 11, "F1_dir": 0.5128, "F1_UP_FORT": 0.234, "F1_DOWN_FORT": 0.3008, "train_start": "2000-11-16", "sampler": "SMOTETomek", "best_params": "{}", "features": ["heston_xi__minus__COF_CapitalOne_ret_5d", "XLY_Disc_zscore_60d__prod__ADM_ArcherDaniels_ret_1d", "US1Y_Rate_ret_20d", "heston_xi__minus__EQIX_Equinix_vol_20d", "VRP", "heston_xi__div__ASML_ASML_vol_20d", "vix_zscore_10d", "IBEX_Spain_ret_20d__macross__ROST_RossStores_zscore_60d", "ROST_RossStores_ret_1d__ret5x__EQIX_Equinix_vol_20d", "ROST_RossStores_ret_1d__macross__WMT_ret_20d", "ADM_ArcherDaniels_ret_1d__ret5x__ADBE_vol_20d"], "is_new": false}, {"model_id": "v1_h1_NORMAL_LightGBM_N12", "algo": "LightGBM", "regime": "NORMAL", "horizon": 1, "n_features": 12, "F1_dir": 0.5374, "F1_UP_FORT": 0.2997, "F1_DOWN_FORT": 0.3333, "train_start": "2000-11-16", "sampler": "SMOTETomek", "best_params": "{}", "features": ["heston_xi__minus__COF_CapitalOne_ret_5d", "XLY_Disc_zscore_60d__prod__ADM_ArcherDaniels_ret_1d", "US1Y_Rate_ret_20d", "heston_xi__minus__EQIX_Equinix_vol_20d", "VRP", "heston_xi__div__ASML_ASML_vol_20d", "vix_zscore_10d", "IBEX_Spain_ret_20d__macross__ROST_RossStores_zscore_60d", "ROST_RossStores_ret_1d__ret5x__EQIX_Equinix_vol_20d", "ROST_RossStores_ret_1d__macross__WMT_ret_20d", "ADM_ArcherDaniels_ret_1d__ret5x__ADBE_vol_20d", "NFCI_ret_5d__minus__ORCL_ret_1d"], "is_new": false}, {"model_id": "v1_h1_NORMAL_LightGBM_N13", "algo": "LightGBM", "regime": "NORMAL", "horizon": 1, "n_features": 13, "F1_dir": 0.5492, "F1_UP_FORT": 0.2733, "F1_DOWN_FORT": 0.3472, "train_start": "2000-11-16", "sampler": "SMOTETomek", "best_params": "{}", "features": ["heston_xi__minus__COF_CapitalOne_ret_5d", "XLY_Disc_zscore_60d__prod__ADM_ArcherDaniels_ret_1d", "US1Y_Rate_ret_20d", "heston_xi__minus__EQIX_Equinix_vol_20d", "VRP", "heston_xi__div__ASML_ASML_vol_20d", "vix_zscore_10d", "IBEX_Spain_ret_20d__macross__ROST_RossStores_zscore_60d", "ROST_RossStores_ret_1d__ret5x__EQIX_Equinix_vol_20d", "ROST_RossStores_ret_1d__macross__WMT_ret_20d", "ADM_ArcherDaniels_ret_1d__ret5x__ADBE_vol_20d", "NFCI_ret_5d__minus__ORCL_ret_1d", "MO_AltriaMG_ret_1d"], "is_new": false}, {"model_id": "v1_h1_NORMAL_LightGBM_N14", "algo": "LightGBM", "regime": "NORMAL", "horizon": 1, "n_features": 14, "F1_dir": 0.5421, "F1_UP_FORT": 0.312, "F1_DOWN_FORT": 0.3467, "train_start": "2000-11-16", "sampler": "SMOTETomek", "best_params": "{}", "features": ["heston_xi__minus__COF_CapitalOne_ret_5d", "XLY_Disc_zscore_60d__prod__ADM_ArcherDaniels_ret_1d", "US1Y_Rate_ret_20d", "heston_xi__minus__EQIX_Equinix_vol_20d", "VRP", "heston_xi__div__ASML_ASML_vol_20d", "vix_zscore_10d", "IBEX_Spain_ret_20d__macross__ROST_RossStores_zscore_60d", "ROST_RossStores_ret_1d__ret5x__EQIX_Equinix_vol_20d", "ROST_RossStores_ret_1d__macross__WMT_ret_20d", "ADM_ArcherDaniels_ret_1d__ret5x__ADBE_vol_20d", "NFCI_ret_5d__minus__ORCL_ret_1d", "MO_AltriaMG_ret_1d", "NFCI_ret_5d__zrel__ORCL_ret_1d"], "is_new": false}, {"model_id": "v1_h1_NORMAL_LightGBM_N15", "algo": "LightGBM", "regime": "NORMAL", "horizon": 1, "n_features": 15, "F1_dir": 0.5782, "F1_UP_FORT": 0.3456, "F1_DOWN_FORT": 0.3215, "train_start": "2000-11-16", "sampler": "SMOTETomek", "best_params": "{}", "features": ["heston_xi__minus__COF_CapitalOne_ret_5d", "XLY_Disc_zscore_60d__prod__ADM_ArcherDaniels_ret_1d", "US1Y_Rate_ret_20d", "heston_xi__minus__EQIX_Equinix_vol_20d", "VRP", "heston_xi__div__ASML_ASML_vol_20d", "vix_zscore_10d", "IBEX_Spain_ret_20d__macross__ROST_RossStores_zscore_60d", "ROST_RossStores_ret_1d__ret5x__EQIX_Equinix_vol_20d", "ROST_RossStores_ret_1d__macross__WMT_ret_20d", "ADM_ArcherDaniels_ret_1d__ret5x__ADBE_vol_20d", "NFCI_ret_5d__minus__ORCL_ret_1d", "MO_AltriaMG_ret_1d", "NFCI_ret_5d__zrel__ORCL_ret_1d", "ROST_RossStores_ret_1d__zrel__ORCL_ret_1d"], "is_new": false}, {"model_id": "v1_h1_NORMAL_LightGBM_N16", "algo": "LightGBM", "regime": "NORMAL", "horizon": 1, "n_features": 16, "F1_dir": 0.5521, "F1_UP_FORT": 0.2955, "F1_DOWN_FORT": 0.3032, "train_start": "2000-11-16", "sampler": "SMOTETomek", "best_params": "{}", "features": ["heston_xi__minus__COF_CapitalOne_ret_5d", "XLY_Disc_zscore_60d__prod__ADM_ArcherDaniels_ret_1d", "US1Y_Rate_ret_20d", "heston_xi__minus__EQIX_Equinix_vol_20d", "VRP", "heston_xi__div__ASML_ASML_vol_20d", "vix_zscore_10d", "IBEX_Spain_ret_20d__macross__ROST_RossStores_zscore_60d", "ROST_RossStores_ret_1d__ret5x__EQIX_Equinix_vol_20d", "ROST_RossStores_ret_1d__macross__WMT_ret_20d", "ADM_ArcherDaniels_ret_1d__ret5x__ADBE_vol_20d", "NFCI_ret_5d__minus__ORCL_ret_1d", "MO_AltriaMG_ret_1d", "NFCI_ret_5d__zrel__ORCL_ret_1d", "ROST_RossStores_ret_1d__zrel__ORCL_ret_1d", "heston_xi__minus__ADBE_vol_20d"], "is_new": false}, {"model_id": "v1_h1_NORMAL_LightGBM_N17", "algo": "LightGBM", "regime": "NORMAL", "horizon": 1, "n_features": 17, "F1_dir": 0.5793, "F1_UP_FORT": 0.3231, "F1_DOWN_FORT": 0.3333, "train_start": "2000-11-16", "sampler": "SMOTETomek", "best_params": "{}", "features": ["heston_xi__minus__COF_CapitalOne_ret_5d", "XLY_Disc_zscore_60d__prod__ADM_ArcherDaniels_ret_1d", "US1Y_Rate_ret_20d", "heston_xi__minus__EQIX_Equinix_vol_20d", "VRP", "heston_xi__div__ASML_ASML_vol_20d", "vix_zscore_10d", "IBEX_Spain_ret_20d__macross__ROST_RossStores_zscore_60d", "ROST_RossStores_ret_1d__ret5x__EQIX_Equinix_vol_20d", "ROST_RossStores_ret_1d__macross__WMT_ret_20d", "ADM_ArcherDaniels_ret_1d__ret5x__ADBE_vol_20d", "NFCI_ret_5d__minus__ORCL_ret_1d", "MO_AltriaMG_ret_1d", "NFCI_ret_5d__zrel__ORCL_ret_1d", "ROST_RossStores_ret_1d__zrel__ORCL_ret_1d", "heston_xi__minus__ADBE_vol_20d", "IBEX_Spain_ret_20d__div__ROST_RossStores_zscore_60d"], "is_new": false}, {"model_id": "v1_h1_NORMAL_LightGBM_N18", "algo": "LightGBM", "regime": "NORMAL", "horizon": 1, "n_features": 18, "F1_dir": 0.5619, "F1_UP_FORT": 0.2992, "F1_DOWN_FORT": 0.336, "train_start": "2000-11-16", "sampler": "SMOTETomek", "best_params": "{}", "features": ["heston_xi__minus__COF_CapitalOne_ret_5d", "XLY_Disc_zscore_60d__prod__ADM_ArcherDaniels_ret_1d", "US1Y_Rate_ret_20d", "heston_xi__minus__EQIX_Equinix_vol_20d", "VRP", "heston_xi__div__ASML_ASML_vol_20d", "vix_zscore_10d", "IBEX_Spain_ret_20d__macross__ROST_RossStores_zscore_60d", "ROST_RossStores_ret_1d__ret5x__EQIX_Equinix_vol_20d", "ROST_RossStores_ret_1d__macross__WMT_ret_20d", "ADM_ArcherDaniels_ret_1d__ret5x__ADBE_vol_20d", "NFCI_ret_5d__minus__ORCL_ret_1d", "MO_AltriaMG_ret_1d", "NFCI_ret_5d__zrel__ORCL_ret_1d", "ROST_RossStores_ret_1d__zrel__ORCL_ret_1d", "heston_xi__minus__ADBE_vol_20d", "IBEX_Spain_ret_20d__div__ROST_RossStores_zscore_60d", "ES_Evergy_ret_1d"], "is_new": false}, {"model_id": "v1_h1_NORMAL_LightGBM_N19", "algo": "LightGBM", "regime": "NORMAL", "horizon": 1, "n_features": 19, "F1_dir": 0.5171, "F1_UP_FORT": 0.2881, "F1_DOWN_FORT": 0.2895, "train_start": "2000-11-16", "sampler": "SMOTETomek", "best_params": "{}", "features": ["heston_xi__minus__COF_CapitalOne_ret_5d", "XLY_Disc_zscore_60d__prod__ADM_ArcherDaniels_ret_1d", "US1Y_Rate_ret_20d", "heston_xi__minus__EQIX_Equinix_vol_20d", "VRP", "heston_xi__div__ASML_ASML_vol_20d", "vix_zscore_10d", "IBEX_Spain_ret_20d__macross__ROST_RossStores_zscore_60d", "ROST_RossStores_ret_1d__ret5x__EQIX_Equinix_vol_20d", "ROST_RossStores_ret_1d__macross__WMT_ret_20d", "ADM_ArcherDaniels_ret_1d__ret5x__ADBE_vol_20d", "NFCI_ret_5d__minus__ORCL_ret_1d", "MO_AltriaMG_ret_1d", "NFCI_ret_5d__zrel__ORCL_ret_1d", "ROST_RossStores_ret_1d__zrel__ORCL_ret_1d", "heston_xi__minus__ADBE_vol_20d", "IBEX_Spain_ret_20d__div__ROST_RossStores_zscore_60d", "ES_Evergy_ret_1d", "MSTR_Bitcoin3_ret_1d"], "is_new": false}, {"model_id": "v1_h1_NORMAL_LightGBM_N20", "algo": "LightGBM", "regime": "NORMAL", "horizon": 1, "n_features": 20, "F1_dir": 0.5469, "F1_UP_FORT": 0.3607, "F1_DOWN_FORT": 0.3027, "train_start": "2000-11-16", "sampler": "SMOTETomek", "best_params": "{}", "features": ["heston_xi__minus__COF_CapitalOne_ret_5d", "XLY_Disc_zscore_60d__prod__ADM_ArcherDaniels_ret_1d", "US1Y_Rate_ret_20d", "heston_xi__minus__EQIX_Equinix_vol_20d", "VRP", "heston_xi__div__ASML_ASML_vol_20d", "vix_zscore_10d", "IBEX_Spain_ret_20d__macross__ROST_RossStores_zscore_60d", "ROST_RossStores_ret_1d__ret5x__EQIX_Equinix_vol_20d", "ROST_RossStores_ret_1d__macross__WMT_ret_20d", "ADM_ArcherDaniels_ret_1d__ret5x__ADBE_vol_20d", "NFCI_ret_5d__minus__ORCL_ret_1d", "MO_AltriaMG_ret_1d", "NFCI_ret_5d__zrel__ORCL_ret_1d", "ROST_RossStores_ret_1d__zrel__ORCL_ret_1d", "heston_xi__minus__ADBE_vol_20d", "IBEX_Spain_ret_20d__div__ROST_RossStores_zscore_60d", "ES_Evergy_ret_1d", "MSTR_Bitcoin3_ret_1d", "LMT_LockheedMartin_ret_1d"], "is_new": false}, {"model_id": "v1_h1_NORMAL_LightGBM_N21", "algo": "LightGBM", "regime": "NORMAL", "horizon": 1, "n_features": 21, "F1_dir": 0.5323, "F1_UP_FORT": 0.3233, "F1_DOWN_FORT": 0.2946, "train_start": "2000-11-16", "sampler": "SMOTETomek", "best_params": "{}", "features": ["heston_xi__minus__COF_CapitalOne_ret_5d", "XLY_Disc_zscore_60d__prod__ADM_ArcherDaniels_ret_1d", "US1Y_Rate_ret_20d", "heston_xi__minus__EQIX_Equinix_vol_20d", "VRP", "heston_xi__div__ASML_ASML_vol_20d", "vix_zscore_10d", "IBEX_Spain_ret_20d__macross__ROST_RossStores_zscore_60d", "ROST_RossStores_ret_1d__ret5x__EQIX_Equinix_vol_20d", "ROST_RossStores_ret_1d__macross__WMT_ret_20d", "ADM_ArcherDaniels_ret_1d__ret5x__ADBE_vol_20d", "NFCI_ret_5d__minus__ORCL_ret_1d", "MO_AltriaMG_ret_1d", "NFCI_ret_5d__zrel__ORCL_ret_1d", "ROST_RossStores_ret_1d__zrel__ORCL_ret_1d", "heston_xi__minus__ADBE_vol_20d", "IBEX_Spain_ret_20d__div__ROST_RossStores_zscore_60d", "ES_Evergy_ret_1d", "MSTR_Bitcoin3_ret_1d", "LMT_LockheedMartin_ret_1d", "INTC_ret_1d"], "is_new": false}, {"model_id": "v1_h1_NORMAL_LightGBM_N22", "algo": "LightGBM", "regime": "NORMAL", "horizon": 1, "n_features": 22, "F1_dir": 0.5168, "F1_UP_FORT": 0.2997, "F1_DOWN_FORT": 0.2985, "train_start": "2000-11-16", "sampler": "SMOTETomek", "best_params": "{}", "features": ["heston_xi__minus__COF_CapitalOne_ret_5d", "XLY_Disc_zscore_60d__prod__ADM_ArcherDaniels_ret_1d", "US1Y_Rate_ret_20d", "heston_xi__minus__EQIX_Equinix_vol_20d", "VRP", "heston_xi__div__ASML_ASML_vol_20d", "vix_zscore_10d", "IBEX_Spain_ret_20d__macross__ROST_RossStores_zscore_60d", "ROST_RossStores_ret_1d__ret5x__EQIX_Equinix_vol_20d", "ROST_RossStores_ret_1d__macross__WMT_ret_20d", "ADM_ArcherDaniels_ret_1d__ret5x__ADBE_vol_20d", "NFCI_ret_5d__minus__ORCL_ret_1d", "MO_AltriaMG_ret_1d", "NFCI_ret_5d__zrel__ORCL_ret_1d", "ROST_RossStores_ret_1d__zrel__ORCL_ret_1d", "heston_xi__minus__ADBE_vol_20d", "IBEX_Spain_ret_20d__div__ROST_RossStores_zscore_60d", "ES_Evergy_ret_1d", "MSTR_Bitcoin3_ret_1d", "LMT_LockheedMartin_ret_1d", "INTC_ret_1d", "VIX_Price_zscore_60d__minus__ROST_RossStores_zscore_60d"], "is_new": false}, {"model_id": "v1_h1_NORMAL_LightGBM_N23", "algo": "LightGBM", "regime": "NORMAL", "horizon": 1, "n_features": 23, "F1_dir": 0.5119, "F1_UP_FORT": 0.2943, "F1_DOWN_FORT": 0.3196, "train_start": "2000-11-16", "sampler": "SMOTETomek", "best_params": "{}", "features": ["heston_xi__minus__COF_CapitalOne_ret_5d", "XLY_Disc_zscore_60d__prod__ADM_ArcherDaniels_ret_1d", "US1Y_Rate_ret_20d", "heston_xi__minus__EQIX_Equinix_vol_20d", "VRP", "heston_xi__div__ASML_ASML_vol_20d", "vix_zscore_10d", "IBEX_Spain_ret_20d__macross__ROST_RossStores_zscore_60d", "ROST_RossStores_ret_1d__ret5x__EQIX_Equinix_vol_20d", "ROST_RossStores_ret_1d__macross__WMT_ret_20d", "ADM_ArcherDaniels_ret_1d__ret5x__ADBE_vol_20d", "NFCI_ret_5d__minus__ORCL_ret_1d", "MO_AltriaMG_ret_1d", "NFCI_ret_5d__zrel__ORCL_ret_1d", "ROST_RossStores_ret_1d__zrel__ORCL_ret_1d", "heston_xi__minus__ADBE_vol_20d", "IBEX_Spain_ret_20d__div__ROST_RossStores_zscore_60d", "ES_Evergy_ret_1d", "MSTR_Bitcoin3_ret_1d", "LMT_LockheedMartin_ret_1d", "INTC_ret_1d", "VIX_Price_zscore_60d__minus__ROST_RossStores_zscore_60d", "VIX_Price_zscore_60d__prod__WMT_ret_20d"], "is_new": false}, {"model_id": "v1_h1_NORMAL_LightGBM_N24", "algo": "LightGBM", "regime": "NORMAL", "horizon": 1, "n_features": 24, "F1_dir": 0.5103, "F1_UP_FORT": 0.3324, "F1_DOWN_FORT": 0.3125, "train_start": "2000-11-16", "sampler": "SMOTETomek", "best_params": "{}", "features": ["heston_xi__minus__COF_CapitalOne_ret_5d", "XLY_Disc_zscore_60d__prod__ADM_ArcherDaniels_ret_1d", "US1Y_Rate_ret_20d", "heston_xi__minus__EQIX_Equinix_vol_20d", "VRP", "heston_xi__div__ASML_ASML_vol_20d", "vix_zscore_10d", "IBEX_Spain_ret_20d__macross__ROST_RossStores_zscore_60d", "ROST_RossStores_ret_1d__ret5x__EQIX_Equinix_vol_20d", "ROST_RossStores_ret_1d__macross__WMT_ret_20d", "ADM_ArcherDaniels_ret_1d__ret5x__ADBE_vol_20d", "NFCI_ret_5d__minus__ORCL_ret_1d", "MO_AltriaMG_ret_1d", "NFCI_ret_5d__zrel__ORCL_ret_1d", "ROST_RossStores_ret_1d__zrel__ORCL_ret_1d", "heston_xi__minus__ADBE_vol_20d", "IBEX_Spain_ret_20d__div__ROST_RossStores_zscore_60d", "ES_Evergy_ret_1d", "MSTR_Bitcoin3_ret_1d", "LMT_LockheedMartin_ret_1d", "INTC_ret_1d", "VIX_Price_zscore_60d__minus__ROST_RossStores_zscore_60d", "VIX_Price_zscore_60d__prod__WMT_ret_20d", "heston_xi__zrel__XLY_Disc_zscore_60d"], "is_new": false}, {"model_id": "v1_h1_NORMAL_LightGBM_N25", "algo": "LightGBM", "regime": "NORMAL", "horizon": 1, "n_features": 25, "F1_dir": 0.5153, "F1_UP_FORT": 0.2791, "F1_DOWN_FORT": 0.3164, "train_start": "2000-11-16", "sampler": "SMOTETomek", "best_params": "{}", "features": ["heston_xi__minus__COF_CapitalOne_ret_5d", "XLY_Disc_zscore_60d__prod__ADM_ArcherDaniels_ret_1d", "US1Y_Rate_ret_20d", "heston_xi__minus__EQIX_Equinix_vol_20d", "VRP", "heston_xi__div__ASML_ASML_vol_20d", "vix_zscore_10d", "IBEX_Spain_ret_20d__macross__ROST_RossStores_zscore_60d", "ROST_RossStores_ret_1d__ret5x__EQIX_Equinix_vol_20d", "ROST_RossStores_ret_1d__macross__WMT_ret_20d", "ADM_ArcherDaniels_ret_1d__ret5x__ADBE_vol_20d", "NFCI_ret_5d__minus__ORCL_ret_1d", "MO_AltriaMG_ret_1d", "NFCI_ret_5d__zrel__ORCL_ret_1d", "ROST_RossStores_ret_1d__zrel__ORCL_ret_1d", "heston_xi__minus__ADBE_vol_20d", "IBEX_Spain_ret_20d__div__ROST_RossStores_zscore_60d", "ES_Evergy_ret_1d", "MSTR_Bitcoin3_ret_1d", "LMT_LockheedMartin_ret_1d", "INTC_ret_1d", "VIX_Price_zscore_60d__minus__ROST_RossStores_zscore_60d", "VIX_Price_zscore_60d__prod__WMT_ret_20d", "heston_xi__zrel__XLY_Disc_zscore_60d", "YUM_YumBrands_ret_1d__minus__WMT_ret_20d"], "is_new": false}, {"model_id": "v1_h1_NORMAL_GradientBoosting_N7", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 1, "n_features": 7, "F1_dir": 0.5288, "F1_UP_FORT": 0.2872, "F1_DOWN_FORT": 0.294, "train_start": "2000-11-16", "sampler": "SMOTETomek", "best_params": "{}", "features": ["heston_xi__minus__COF_CapitalOne_ret_5d", "XLY_Disc_zscore_60d__prod__ADM_ArcherDaniels_ret_1d", "US1Y_Rate_ret_20d", "heston_xi__minus__EQIX_Equinix_vol_20d", "VRP", "heston_xi__div__ASML_ASML_vol_20d", "vix_zscore_10d"], "is_new": false}, {"model_id": "v1_h1_NORMAL_GradientBoosting_N8", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 1, "n_features": 8, "F1_dir": 0.5134, "F1_UP_FORT": 0.2216, "F1_DOWN_FORT": 0.2595, "train_start": "2000-11-16", "sampler": "SMOTETomek", "best_params": "{}", "features": ["heston_xi__minus__COF_CapitalOne_ret_5d", "XLY_Disc_zscore_60d__prod__ADM_ArcherDaniels_ret_1d", "US1Y_Rate_ret_20d", "heston_xi__minus__EQIX_Equinix_vol_20d", "VRP", "heston_xi__div__ASML_ASML_vol_20d", "vix_zscore_10d", "IBEX_Spain_ret_20d__macross__ROST_RossStores_zscore_60d"], "is_new": false}, {"model_id": "v1_h1_NORMAL_GradientBoosting_N9", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 1, "n_features": 9, "F1_dir": 0.5263, "F1_UP_FORT": 0.2178, "F1_DOWN_FORT": 0.2787, "train_start": "2000-11-16", "sampler": "SMOTETomek", "best_params": "{}", "features": ["heston_xi__minus__COF_CapitalOne_ret_5d", "XLY_Disc_zscore_60d__prod__ADM_ArcherDaniels_ret_1d", "US1Y_Rate_ret_20d", "heston_xi__minus__EQIX_Equinix_vol_20d", "VRP", "heston_xi__div__ASML_ASML_vol_20d", "vix_zscore_10d", "IBEX_Spain_ret_20d__macross__ROST_RossStores_zscore_60d", "ROST_RossStores_ret_1d__ret5x__EQIX_Equinix_vol_20d"], "is_new": false}, {"model_id": "v1_h1_NORMAL_GradientBoosting_N10", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 1, "n_features": 10, "F1_dir": 0.5112, "F1_UP_FORT": 0.2149, "F1_DOWN_FORT": 0.268, "train_start": "2000-11-16", "sampler": "SMOTETomek", "best_params": "{}", "features": ["heston_xi__minus__COF_CapitalOne_ret_5d", "XLY_Disc_zscore_60d__prod__ADM_ArcherDaniels_ret_1d", "US1Y_Rate_ret_20d", "heston_xi__minus__EQIX_Equinix_vol_20d", "VRP", "heston_xi__div__ASML_ASML_vol_20d", "vix_zscore_10d", "IBEX_Spain_ret_20d__macross__ROST_RossStores_zscore_60d", "ROST_RossStores_ret_1d__ret5x__EQIX_Equinix_vol_20d", "ROST_RossStores_ret_1d__macross__WMT_ret_20d"], "is_new": false}, {"model_id": "v1_h1_NORMAL_GradientBoosting_N11", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 1, "n_features": 11, "F1_dir": 0.5337, "F1_UP_FORT": 0.2377, "F1_DOWN_FORT": 0.285, "train_start": "2000-11-16", "sampler": "SMOTETomek", "best_params": "{}", "features": ["heston_xi__minus__COF_CapitalOne_ret_5d", "XLY_Disc_zscore_60d__prod__ADM_ArcherDaniels_ret_1d", "US1Y_Rate_ret_20d", "heston_xi__minus__EQIX_Equinix_vol_20d", "VRP", "heston_xi__div__ASML_ASML_vol_20d", "vix_zscore_10d", "IBEX_Spain_ret_20d__macross__ROST_RossStores_zscore_60d", "ROST_RossStores_ret_1d__ret5x__EQIX_Equinix_vol_20d", "ROST_RossStores_ret_1d__macross__WMT_ret_20d", "ADM_ArcherDaniels_ret_1d__ret5x__ADBE_vol_20d"], "is_new": false}, {"model_id": "v1_h1_NORMAL_GradientBoosting_N12", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 1, "n_features": 12, "F1_dir": 0.519, "F1_UP_FORT": 0.2356, "F1_DOWN_FORT": 0.2827, "train_start": "2000-11-16", "sampler": "SMOTETomek", "best_params": "{}", "features": ["heston_xi__minus__COF_CapitalOne_ret_5d", "XLY_Disc_zscore_60d__prod__ADM_ArcherDaniels_ret_1d", "US1Y_Rate_ret_20d", "heston_xi__minus__EQIX_Equinix_vol_20d", "VRP", "heston_xi__div__ASML_ASML_vol_20d", "vix_zscore_10d", "IBEX_Spain_ret_20d__macross__ROST_RossStores_zscore_60d", "ROST_RossStores_ret_1d__ret5x__EQIX_Equinix_vol_20d", "ROST_RossStores_ret_1d__macross__WMT_ret_20d", "ADM_ArcherDaniels_ret_1d__ret5x__ADBE_vol_20d", "NFCI_ret_5d__minus__ORCL_ret_1d"], "is_new": false}, {"model_id": "v1_h1_NORMAL_GradientBoosting_N13", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 1, "n_features": 13, "F1_dir": 0.5338, "F1_UP_FORT": 0.2629, "F1_DOWN_FORT": 0.3212, "train_start": "2000-11-16", "sampler": "SMOTETomek", "best_params": "{}", "features": ["heston_xi__minus__COF_CapitalOne_ret_5d", "XLY_Disc_zscore_60d__prod__ADM_ArcherDaniels_ret_1d", "US1Y_Rate_ret_20d", "heston_xi__minus__EQIX_Equinix_vol_20d", "VRP", "heston_xi__div__ASML_ASML_vol_20d", "vix_zscore_10d", "IBEX_Spain_ret_20d__macross__ROST_RossStores_zscore_60d", "ROST_RossStores_ret_1d__ret5x__EQIX_Equinix_vol_20d", "ROST_RossStores_ret_1d__macross__WMT_ret_20d", "ADM_ArcherDaniels_ret_1d__ret5x__ADBE_vol_20d", "NFCI_ret_5d__minus__ORCL_ret_1d", "MO_AltriaMG_ret_1d"], "is_new": false}, {"model_id": "v1_h1_NORMAL_GradientBoosting_N14", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 1, "n_features": 14, "F1_dir": 0.5463, "F1_UP_FORT": 0.2621, "F1_DOWN_FORT": 0.3147, "train_start": "2000-11-16", "sampler": "SMOTETomek", "best_params": "{}", "features": ["heston_xi__minus__COF_CapitalOne_ret_5d", "XLY_Disc_zscore_60d__prod__ADM_ArcherDaniels_ret_1d", "US1Y_Rate_ret_20d", "heston_xi__minus__EQIX_Equinix_vol_20d", "VRP", "heston_xi__div__ASML_ASML_vol_20d", "vix_zscore_10d", "IBEX_Spain_ret_20d__macross__ROST_RossStores_zscore_60d", "ROST_RossStores_ret_1d__ret5x__EQIX_Equinix_vol_20d", "ROST_RossStores_ret_1d__macross__WMT_ret_20d", "ADM_ArcherDaniels_ret_1d__ret5x__ADBE_vol_20d", "NFCI_ret_5d__minus__ORCL_ret_1d", "MO_AltriaMG_ret_1d", "NFCI_ret_5d__zrel__ORCL_ret_1d"], "is_new": false}, {"model_id": "v1_h1_NORMAL_GradientBoosting_N15", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 1, "n_features": 15, "F1_dir": 0.5617, "F1_UP_FORT": 0.3027, "F1_DOWN_FORT": 0.3169, "train_start": "2000-11-16", "sampler": "SMOTETomek", "best_params": "{}", "features": ["heston_xi__minus__COF_CapitalOne_ret_5d", "XLY_Disc_zscore_60d__prod__ADM_ArcherDaniels_ret_1d", "US1Y_Rate_ret_20d", "heston_xi__minus__EQIX_Equinix_vol_20d", "VRP", "heston_xi__div__ASML_ASML_vol_20d", "vix_zscore_10d", "IBEX_Spain_ret_20d__macross__ROST_RossStores_zscore_60d", "ROST_RossStores_ret_1d__ret5x__EQIX_Equinix_vol_20d", "ROST_RossStores_ret_1d__macross__WMT_ret_20d", "ADM_ArcherDaniels_ret_1d__ret5x__ADBE_vol_20d", "NFCI_ret_5d__minus__ORCL_ret_1d", "MO_AltriaMG_ret_1d", "NFCI_ret_5d__zrel__ORCL_ret_1d", "ROST_RossStores_ret_1d__zrel__ORCL_ret_1d"], "is_new": false}, {"model_id": "v1_h1_NORMAL_GradientBoosting_N16", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 1, "n_features": 16, "F1_dir": 0.5582, "F1_UP_FORT": 0.3063, "F1_DOWN_FORT": 0.3188, "train_start": "2000-11-16", "sampler": "SMOTETomek", "best_params": "{}", "features": ["heston_xi__minus__COF_CapitalOne_ret_5d", "XLY_Disc_zscore_60d__prod__ADM_ArcherDaniels_ret_1d", "US1Y_Rate_ret_20d", "heston_xi__minus__EQIX_Equinix_vol_20d", "VRP", "heston_xi__div__ASML_ASML_vol_20d", "vix_zscore_10d", "IBEX_Spain_ret_20d__macross__ROST_RossStores_zscore_60d", "ROST_RossStores_ret_1d__ret5x__EQIX_Equinix_vol_20d", "ROST_RossStores_ret_1d__macross__WMT_ret_20d", "ADM_ArcherDaniels_ret_1d__ret5x__ADBE_vol_20d", "NFCI_ret_5d__minus__ORCL_ret_1d", "MO_AltriaMG_ret_1d", "NFCI_ret_5d__zrel__ORCL_ret_1d", "ROST_RossStores_ret_1d__zrel__ORCL_ret_1d", "heston_xi__minus__ADBE_vol_20d"], "is_new": false}, {"model_id": "v1_h1_NORMAL_GradientBoosting_N17", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 1, "n_features": 17, "F1_dir": 0.5771, "F1_UP_FORT": 0.3055, "F1_DOWN_FORT": 0.3333, "train_start": "2000-11-16", "sampler": "SMOTETomek", "best_params": "{}", "features": ["heston_xi__minus__COF_CapitalOne_ret_5d", "XLY_Disc_zscore_60d__prod__ADM_ArcherDaniels_ret_1d", "US1Y_Rate_ret_20d", "heston_xi__minus__EQIX_Equinix_vol_20d", "VRP", "heston_xi__div__ASML_ASML_vol_20d", "vix_zscore_10d", "IBEX_Spain_ret_20d__macross__ROST_RossStores_zscore_60d", "ROST_RossStores_ret_1d__ret5x__EQIX_Equinix_vol_20d", "ROST_RossStores_ret_1d__macross__WMT_ret_20d", "ADM_ArcherDaniels_ret_1d__ret5x__ADBE_vol_20d", "NFCI_ret_5d__minus__ORCL_ret_1d", "MO_AltriaMG_ret_1d", "NFCI_ret_5d__zrel__ORCL_ret_1d", "ROST_RossStores_ret_1d__zrel__ORCL_ret_1d", "heston_xi__minus__ADBE_vol_20d", "IBEX_Spain_ret_20d__div__ROST_RossStores_zscore_60d"], "is_new": false}, {"model_id": "v1_h1_NORMAL_GradientBoosting_N18", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 1, "n_features": 18, "F1_dir": 0.5432, "F1_UP_FORT": 0.2899, "F1_DOWN_FORT": 0.3234, "train_start": "2000-11-16", "sampler": "SMOTETomek", "best_params": "{}", "features": ["heston_xi__minus__COF_CapitalOne_ret_5d", "XLY_Disc_zscore_60d__prod__ADM_ArcherDaniels_ret_1d", "US1Y_Rate_ret_20d", "heston_xi__minus__EQIX_Equinix_vol_20d", "VRP", "heston_xi__div__ASML_ASML_vol_20d", "vix_zscore_10d", "IBEX_Spain_ret_20d__macross__ROST_RossStores_zscore_60d", "ROST_RossStores_ret_1d__ret5x__EQIX_Equinix_vol_20d", "ROST_RossStores_ret_1d__macross__WMT_ret_20d", "ADM_ArcherDaniels_ret_1d__ret5x__ADBE_vol_20d", "NFCI_ret_5d__minus__ORCL_ret_1d", "MO_AltriaMG_ret_1d", "NFCI_ret_5d__zrel__ORCL_ret_1d", "ROST_RossStores_ret_1d__zrel__ORCL_ret_1d", "heston_xi__minus__ADBE_vol_20d", "IBEX_Spain_ret_20d__div__ROST_RossStores_zscore_60d", "ES_Evergy_ret_1d"], "is_new": false}, {"model_id": "v1_h1_NORMAL_GradientBoosting_N19", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 1, "n_features": 19, "F1_dir": 0.541, "F1_UP_FORT": 0.3314, "F1_DOWN_FORT": 0.3185, "train_start": "2000-11-16", "sampler": "SMOTETomek", "best_params": "{}", "features": ["heston_xi__minus__COF_CapitalOne_ret_5d", "XLY_Disc_zscore_60d__prod__ADM_ArcherDaniels_ret_1d", "US1Y_Rate_ret_20d", "heston_xi__minus__EQIX_Equinix_vol_20d", "VRP", "heston_xi__div__ASML_ASML_vol_20d", "vix_zscore_10d", "IBEX_Spain_ret_20d__macross__ROST_RossStores_zscore_60d", "ROST_RossStores_ret_1d__ret5x__EQIX_Equinix_vol_20d", "ROST_RossStores_ret_1d__macross__WMT_ret_20d", "ADM_ArcherDaniels_ret_1d__ret5x__ADBE_vol_20d", "NFCI_ret_5d__minus__ORCL_ret_1d", "MO_AltriaMG_ret_1d", "NFCI_ret_5d__zrel__ORCL_ret_1d", "ROST_RossStores_ret_1d__zrel__ORCL_ret_1d", "heston_xi__minus__ADBE_vol_20d", "IBEX_Spain_ret_20d__div__ROST_RossStores_zscore_60d", "ES_Evergy_ret_1d", "MSTR_Bitcoin3_ret_1d"], "is_new": false}, {"model_id": "v1_h1_NORMAL_GradientBoosting_N20", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 1, "n_features": 20, "F1_dir": 0.5275, "F1_UP_FORT": 0.3008, "F1_DOWN_FORT": 0.294, "train_start": "2000-11-16", "sampler": "SMOTETomek", "best_params": "{}", "features": ["heston_xi__minus__COF_CapitalOne_ret_5d", "XLY_Disc_zscore_60d__prod__ADM_ArcherDaniels_ret_1d", "US1Y_Rate_ret_20d", "heston_xi__minus__EQIX_Equinix_vol_20d", "VRP", "heston_xi__div__ASML_ASML_vol_20d", "vix_zscore_10d", "IBEX_Spain_ret_20d__macross__ROST_RossStores_zscore_60d", "ROST_RossStores_ret_1d__ret5x__EQIX_Equinix_vol_20d", "ROST_RossStores_ret_1d__macross__WMT_ret_20d", "ADM_ArcherDaniels_ret_1d__ret5x__ADBE_vol_20d", "NFCI_ret_5d__minus__ORCL_ret_1d", "MO_AltriaMG_ret_1d", "NFCI_ret_5d__zrel__ORCL_ret_1d", "ROST_RossStores_ret_1d__zrel__ORCL_ret_1d", "heston_xi__minus__ADBE_vol_20d", "IBEX_Spain_ret_20d__div__ROST_RossStores_zscore_60d", "ES_Evergy_ret_1d", "MSTR_Bitcoin3_ret_1d", "LMT_LockheedMartin_ret_1d"], "is_new": false}, {"model_id": "v1_h1_NORMAL_GradientBoosting_N21", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 1, "n_features": 21, "F1_dir": 0.5315, "F1_UP_FORT": 0.3172, "F1_DOWN_FORT": 0.2957, "train_start": "2000-11-16", "sampler": "SMOTETomek", "best_params": "{}", "features": ["heston_xi__minus__COF_CapitalOne_ret_5d", "XLY_Disc_zscore_60d__prod__ADM_ArcherDaniels_ret_1d", "US1Y_Rate_ret_20d", "heston_xi__minus__EQIX_Equinix_vol_20d", "VRP", "heston_xi__div__ASML_ASML_vol_20d", "vix_zscore_10d", "IBEX_Spain_ret_20d__macross__ROST_RossStores_zscore_60d", "ROST_RossStores_ret_1d__ret5x__EQIX_Equinix_vol_20d", "ROST_RossStores_ret_1d__macross__WMT_ret_20d", "ADM_ArcherDaniels_ret_1d__ret5x__ADBE_vol_20d", "NFCI_ret_5d__minus__ORCL_ret_1d", "MO_AltriaMG_ret_1d", "NFCI_ret_5d__zrel__ORCL_ret_1d", "ROST_RossStores_ret_1d__zrel__ORCL_ret_1d", "heston_xi__minus__ADBE_vol_20d", "IBEX_Spain_ret_20d__div__ROST_RossStores_zscore_60d", "ES_Evergy_ret_1d", "MSTR_Bitcoin3_ret_1d", "LMT_LockheedMartin_ret_1d", "INTC_ret_1d"], "is_new": false}, {"model_id": "v1_h1_NORMAL_GradientBoosting_N22", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 1, "n_features": 22, "F1_dir": 0.5402, "F1_UP_FORT": 0.326, "F1_DOWN_FORT": 0.3116, "train_start": "2000-11-16", "sampler": "SMOTETomek", "best_params": "{}", "features": ["heston_xi__minus__COF_CapitalOne_ret_5d", "XLY_Disc_zscore_60d__prod__ADM_ArcherDaniels_ret_1d", "US1Y_Rate_ret_20d", "heston_xi__minus__EQIX_Equinix_vol_20d", "VRP", "heston_xi__div__ASML_ASML_vol_20d", "vix_zscore_10d", "IBEX_Spain_ret_20d__macross__ROST_RossStores_zscore_60d", "ROST_RossStores_ret_1d__ret5x__EQIX_Equinix_vol_20d", "ROST_RossStores_ret_1d__macross__WMT_ret_20d", "ADM_ArcherDaniels_ret_1d__ret5x__ADBE_vol_20d", "NFCI_ret_5d__minus__ORCL_ret_1d", "MO_AltriaMG_ret_1d", "NFCI_ret_5d__zrel__ORCL_ret_1d", "ROST_RossStores_ret_1d__zrel__ORCL_ret_1d", "heston_xi__minus__ADBE_vol_20d", "IBEX_Spain_ret_20d__div__ROST_RossStores_zscore_60d", "ES_Evergy_ret_1d", "MSTR_Bitcoin3_ret_1d", "LMT_LockheedMartin_ret_1d", "INTC_ret_1d", "VIX_Price_zscore_60d__minus__ROST_RossStores_zscore_60d"], "is_new": false}, {"model_id": "v1_h1_NORMAL_GradientBoosting_N23", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 1, "n_features": 23, "F1_dir": 0.5105, "F1_UP_FORT": 0.3125, "F1_DOWN_FORT": 0.3261, "train_start": "2000-11-16", "sampler": "SMOTETomek", "best_params": "{}", "features": ["heston_xi__minus__COF_CapitalOne_ret_5d", "XLY_Disc_zscore_60d__prod__ADM_ArcherDaniels_ret_1d", "US1Y_Rate_ret_20d", "heston_xi__minus__EQIX_Equinix_vol_20d", "VRP", "heston_xi__div__ASML_ASML_vol_20d", "vix_zscore_10d", "IBEX_Spain_ret_20d__macross__ROST_RossStores_zscore_60d", "ROST_RossStores_ret_1d__ret5x__EQIX_Equinix_vol_20d", "ROST_RossStores_ret_1d__macross__WMT_ret_20d", "ADM_ArcherDaniels_ret_1d__ret5x__ADBE_vol_20d", "NFCI_ret_5d__minus__ORCL_ret_1d", "MO_AltriaMG_ret_1d", "NFCI_ret_5d__zrel__ORCL_ret_1d", "ROST_RossStores_ret_1d__zrel__ORCL_ret_1d", "heston_xi__minus__ADBE_vol_20d", "IBEX_Spain_ret_20d__div__ROST_RossStores_zscore_60d", "ES_Evergy_ret_1d", "MSTR_Bitcoin3_ret_1d", "LMT_LockheedMartin_ret_1d", "INTC_ret_1d", "VIX_Price_zscore_60d__minus__ROST_RossStores_zscore_60d", "VIX_Price_zscore_60d__prod__WMT_ret_20d"], "is_new": false}, {"model_id": "v1_h1_NORMAL_GradientBoosting_N24", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 1, "n_features": 24, "F1_dir": 0.5115, "F1_UP_FORT": 0.3115, "F1_DOWN_FORT": 0.329, "train_start": "2000-11-16", "sampler": "SMOTETomek", "best_params": "{}", "features": ["heston_xi__minus__COF_CapitalOne_ret_5d", "XLY_Disc_zscore_60d__prod__ADM_ArcherDaniels_ret_1d", "US1Y_Rate_ret_20d", "heston_xi__minus__EQIX_Equinix_vol_20d", "VRP", "heston_xi__div__ASML_ASML_vol_20d", "vix_zscore_10d", "IBEX_Spain_ret_20d__macross__ROST_RossStores_zscore_60d", "ROST_RossStores_ret_1d__ret5x__EQIX_Equinix_vol_20d", "ROST_RossStores_ret_1d__macross__WMT_ret_20d", "ADM_ArcherDaniels_ret_1d__ret5x__ADBE_vol_20d", "NFCI_ret_5d__minus__ORCL_ret_1d", "MO_AltriaMG_ret_1d", "NFCI_ret_5d__zrel__ORCL_ret_1d", "ROST_RossStores_ret_1d__zrel__ORCL_ret_1d", "heston_xi__minus__ADBE_vol_20d", "IBEX_Spain_ret_20d__div__ROST_RossStores_zscore_60d", "ES_Evergy_ret_1d", "MSTR_Bitcoin3_ret_1d", "LMT_LockheedMartin_ret_1d", "INTC_ret_1d", "VIX_Price_zscore_60d__minus__ROST_RossStores_zscore_60d", "VIX_Price_zscore_60d__prod__WMT_ret_20d", "heston_xi__zrel__XLY_Disc_zscore_60d"], "is_new": false}, {"model_id": "v1_h1_NORMAL_GradientBoosting_N25", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 1, "n_features": 25, "F1_dir": 0.5105, "F1_UP_FORT": 0.2799, "F1_DOWN_FORT": 0.3385, "train_start": "2000-11-16", "sampler": "SMOTETomek", "best_params": "{}", "features": ["heston_xi__minus__COF_CapitalOne_ret_5d", "XLY_Disc_zscore_60d__prod__ADM_ArcherDaniels_ret_1d", "US1Y_Rate_ret_20d", "heston_xi__minus__EQIX_Equinix_vol_20d", "VRP", "heston_xi__div__ASML_ASML_vol_20d", "vix_zscore_10d", "IBEX_Spain_ret_20d__macross__ROST_RossStores_zscore_60d", "ROST_RossStores_ret_1d__ret5x__EQIX_Equinix_vol_20d", "ROST_RossStores_ret_1d__macross__WMT_ret_20d", "ADM_ArcherDaniels_ret_1d__ret5x__ADBE_vol_20d", "NFCI_ret_5d__minus__ORCL_ret_1d", "MO_AltriaMG_ret_1d", "NFCI_ret_5d__zrel__ORCL_ret_1d", "ROST_RossStores_ret_1d__zrel__ORCL_ret_1d", "heston_xi__minus__ADBE_vol_20d", "IBEX_Spain_ret_20d__div__ROST_RossStores_zscore_60d", "ES_Evergy_ret_1d", "MSTR_Bitcoin3_ret_1d", "LMT_LockheedMartin_ret_1d", "INTC_ret_1d", "VIX_Price_zscore_60d__minus__ROST_RossStores_zscore_60d", "VIX_Price_zscore_60d__prod__WMT_ret_20d", "heston_xi__zrel__XLY_Disc_zscore_60d", "YUM_YumBrands_ret_1d__minus__WMT_ret_20d"], "is_new": false}, {"model_id": "v1_h1_NORMAL_RandomForest_N5", "algo": "RandomForest", "regime": "NORMAL", "horizon": 1, "n_features": 5, "F1_dir": 0.5035, "F1_UP_FORT": 0.2448, "F1_DOWN_FORT": 0.3543, "train_start": "2000-11-16", "sampler": "SMOTETomek", "best_params": "{}", "features": ["heston_xi__minus__COF_CapitalOne_ret_5d", "XLY_Disc_zscore_60d__prod__ADM_ArcherDaniels_ret_1d", "US1Y_Rate_ret_20d", "heston_xi__minus__EQIX_Equinix_vol_20d", "VRP"], "is_new": false}, {"model_id": "v1_h1_NORMAL_RandomForest_N8", "algo": "RandomForest", "regime": "NORMAL", "horizon": 1, "n_features": 8, "F1_dir": 0.5095, "F1_UP_FORT": 0.1824, "F1_DOWN_FORT": 0.3636, "train_start": "2000-11-16", "sampler": "SMOTETomek", "best_params": "{}", "features": ["heston_xi__minus__COF_CapitalOne_ret_5d", "XLY_Disc_zscore_60d__prod__ADM_ArcherDaniels_ret_1d", "US1Y_Rate_ret_20d", "heston_xi__minus__EQIX_Equinix_vol_20d", "VRP", "heston_xi__div__ASML_ASML_vol_20d", "vix_zscore_10d", "IBEX_Spain_ret_20d__macross__ROST_RossStores_zscore_60d"], "is_new": false}, {"model_id": "v1_h1_NORMAL_RandomForest_N16", "algo": "RandomForest", "regime": "NORMAL", "horizon": 1, "n_features": 16, "F1_dir": 0.5066, "F1_UP_FORT": 0.2699, "F1_DOWN_FORT": 0.3856, "train_start": "2000-11-16", "sampler": "SMOTETomek", "best_params": "{}", "features": ["heston_xi__minus__COF_CapitalOne_ret_5d", "XLY_Disc_zscore_60d__prod__ADM_ArcherDaniels_ret_1d", "US1Y_Rate_ret_20d", "heston_xi__minus__EQIX_Equinix_vol_20d", "VRP", "heston_xi__div__ASML_ASML_vol_20d", "vix_zscore_10d", "IBEX_Spain_ret_20d__macross__ROST_RossStores_zscore_60d", "ROST_RossStores_ret_1d__ret5x__EQIX_Equinix_vol_20d", "ROST_RossStores_ret_1d__macross__WMT_ret_20d", "ADM_ArcherDaniels_ret_1d__ret5x__ADBE_vol_20d", "NFCI_ret_5d__minus__ORCL_ret_1d", "MO_AltriaMG_ret_1d", "NFCI_ret_5d__zrel__ORCL_ret_1d", "ROST_RossStores_ret_1d__zrel__ORCL_ret_1d", "heston_xi__minus__ADBE_vol_20d"], "is_new": false}, {"model_id": "v1_h1_NORMAL_RandomForest_N23", "algo": "RandomForest", "regime": "NORMAL", "horizon": 1, "n_features": 23, "F1_dir": 0.5104, "F1_UP_FORT": 0.284, "F1_DOWN_FORT": 0.3732, "train_start": "2000-11-16", "sampler": "SMOTETomek", "best_params": "{}", "features": ["heston_xi__minus__COF_CapitalOne_ret_5d", "XLY_Disc_zscore_60d__prod__ADM_ArcherDaniels_ret_1d", "US1Y_Rate_ret_20d", "heston_xi__minus__EQIX_Equinix_vol_20d", "VRP", "heston_xi__div__ASML_ASML_vol_20d", "vix_zscore_10d", "IBEX_Spain_ret_20d__macross__ROST_RossStores_zscore_60d", "ROST_RossStores_ret_1d__ret5x__EQIX_Equinix_vol_20d", "ROST_RossStores_ret_1d__macross__WMT_ret_20d", "ADM_ArcherDaniels_ret_1d__ret5x__ADBE_vol_20d", "NFCI_ret_5d__minus__ORCL_ret_1d", "MO_AltriaMG_ret_1d", "NFCI_ret_5d__zrel__ORCL_ret_1d", "ROST_RossStores_ret_1d__zrel__ORCL_ret_1d", "heston_xi__minus__ADBE_vol_20d", "IBEX_Spain_ret_20d__div__ROST_RossStores_zscore_60d", "ES_Evergy_ret_1d", "MSTR_Bitcoin3_ret_1d", "LMT_LockheedMartin_ret_1d", "INTC_ret_1d", "VIX_Price_zscore_60d__minus__ROST_RossStores_zscore_60d", "VIX_Price_zscore_60d__prod__WMT_ret_20d"], "is_new": false}, {"model_id": "v1_h1_NORMAL_RandomForest_N25", "algo": "RandomForest", "regime": "NORMAL", "horizon": 1, "n_features": 25, "F1_dir": 0.506, "F1_UP_FORT": 0.2687, "F1_DOWN_FORT": 0.3811, "train_start": "2000-11-16", "sampler": "SMOTETomek", "best_params": "{}", "features": ["heston_xi__minus__COF_CapitalOne_ret_5d", "XLY_Disc_zscore_60d__prod__ADM_ArcherDaniels_ret_1d", "US1Y_Rate_ret_20d", "heston_xi__minus__EQIX_Equinix_vol_20d", "VRP", "heston_xi__div__ASML_ASML_vol_20d", "vix_zscore_10d", "IBEX_Spain_ret_20d__macross__ROST_RossStores_zscore_60d", "ROST_RossStores_ret_1d__ret5x__EQIX_Equinix_vol_20d", "ROST_RossStores_ret_1d__macross__WMT_ret_20d", "ADM_ArcherDaniels_ret_1d__ret5x__ADBE_vol_20d", "NFCI_ret_5d__minus__ORCL_ret_1d", "MO_AltriaMG_ret_1d", "NFCI_ret_5d__zrel__ORCL_ret_1d", "ROST_RossStores_ret_1d__zrel__ORCL_ret_1d", "heston_xi__minus__ADBE_vol_20d", "IBEX_Spain_ret_20d__div__ROST_RossStores_zscore_60d", "ES_Evergy_ret_1d", "MSTR_Bitcoin3_ret_1d", "LMT_LockheedMartin_ret_1d", "INTC_ret_1d", "VIX_Price_zscore_60d__minus__ROST_RossStores_zscore_60d", "VIX_Price_zscore_60d__prod__WMT_ret_20d", "heston_xi__zrel__XLY_Disc_zscore_60d", "YUM_YumBrands_ret_1d__minus__WMT_ret_20d"], "is_new": false}, {"model_id": "v1_h1_NORMAL_LogisticRegression_N15", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 1, "n_features": 15, "F1_dir": 0.5154, "F1_UP_FORT": 0.2038, "F1_DOWN_FORT": 0.3731, "train_start": "2000-11-16", "sampler": "SMOTETomek", "best_params": "{}", "features": ["heston_xi__minus__COF_CapitalOne_ret_5d", "XLY_Disc_zscore_60d__prod__ADM_ArcherDaniels_ret_1d", "US1Y_Rate_ret_20d", "heston_xi__minus__EQIX_Equinix_vol_20d", "VRP", "heston_xi__div__ASML_ASML_vol_20d", "vix_zscore_10d", "IBEX_Spain_ret_20d__macross__ROST_RossStores_zscore_60d", "ROST_RossStores_ret_1d__ret5x__EQIX_Equinix_vol_20d", "ROST_RossStores_ret_1d__macross__WMT_ret_20d", "ADM_ArcherDaniels_ret_1d__ret5x__ADBE_vol_20d", "NFCI_ret_5d__minus__ORCL_ret_1d", "MO_AltriaMG_ret_1d", "NFCI_ret_5d__zrel__ORCL_ret_1d", "ROST_RossStores_ret_1d__zrel__ORCL_ret_1d"], "is_new": false}, {"model_id": "v1_h1_NORMAL_LogisticRegression_N16", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 1, "n_features": 16, "F1_dir": 0.5142, "F1_UP_FORT": 0.2038, "F1_DOWN_FORT": 0.3709, "train_start": "2000-11-16", "sampler": "SMOTETomek", "best_params": "{}", "features": ["heston_xi__minus__COF_CapitalOne_ret_5d", "XLY_Disc_zscore_60d__prod__ADM_ArcherDaniels_ret_1d", "US1Y_Rate_ret_20d", "heston_xi__minus__EQIX_Equinix_vol_20d", "VRP", "heston_xi__div__ASML_ASML_vol_20d", "vix_zscore_10d", "IBEX_Spain_ret_20d__macross__ROST_RossStores_zscore_60d", "ROST_RossStores_ret_1d__ret5x__EQIX_Equinix_vol_20d", "ROST_RossStores_ret_1d__macross__WMT_ret_20d", "ADM_ArcherDaniels_ret_1d__ret5x__ADBE_vol_20d", "NFCI_ret_5d__minus__ORCL_ret_1d", "MO_AltriaMG_ret_1d", "NFCI_ret_5d__zrel__ORCL_ret_1d", "ROST_RossStores_ret_1d__zrel__ORCL_ret_1d", "heston_xi__minus__ADBE_vol_20d"], "is_new": false}, {"model_id": "v1_h1_NORMAL_LogisticRegression_N17", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 1, "n_features": 17, "F1_dir": 0.5196, "F1_UP_FORT": 0.2201, "F1_DOWN_FORT": 0.3636, "train_start": "2000-11-16", "sampler": "SMOTETomek", "best_params": "{}", "features": ["heston_xi__minus__COF_CapitalOne_ret_5d", "XLY_Disc_zscore_60d__prod__ADM_ArcherDaniels_ret_1d", "US1Y_Rate_ret_20d", "heston_xi__minus__EQIX_Equinix_vol_20d", "VRP", "heston_xi__div__ASML_ASML_vol_20d", "vix_zscore_10d", "IBEX_Spain_ret_20d__macross__ROST_RossStores_zscore_60d", "ROST_RossStores_ret_1d__ret5x__EQIX_Equinix_vol_20d", "ROST_RossStores_ret_1d__macross__WMT_ret_20d", "ADM_ArcherDaniels_ret_1d__ret5x__ADBE_vol_20d", "NFCI_ret_5d__minus__ORCL_ret_1d", "MO_AltriaMG_ret_1d", "NFCI_ret_5d__zrel__ORCL_ret_1d", "ROST_RossStores_ret_1d__zrel__ORCL_ret_1d", "heston_xi__minus__ADBE_vol_20d", "IBEX_Spain_ret_20d__div__ROST_RossStores_zscore_60d"], "is_new": false}, {"model_id": "v1_h1_NORMAL_LogisticRegression_N18", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 1, "n_features": 18, "F1_dir": 0.5188, "F1_UP_FORT": 0.2609, "F1_DOWN_FORT": 0.3687, "train_start": "2000-11-16", "sampler": "SMOTETomek", "best_params": "{}", "features": ["heston_xi__minus__COF_CapitalOne_ret_5d", "XLY_Disc_zscore_60d__prod__ADM_ArcherDaniels_ret_1d", "US1Y_Rate_ret_20d", "heston_xi__minus__EQIX_Equinix_vol_20d", "VRP", "heston_xi__div__ASML_ASML_vol_20d", "vix_zscore_10d", "IBEX_Spain_ret_20d__macross__ROST_RossStores_zscore_60d", "ROST_RossStores_ret_1d__ret5x__EQIX_Equinix_vol_20d", "ROST_RossStores_ret_1d__macross__WMT_ret_20d", "ADM_ArcherDaniels_ret_1d__ret5x__ADBE_vol_20d", "NFCI_ret_5d__minus__ORCL_ret_1d", "MO_AltriaMG_ret_1d", "NFCI_ret_5d__zrel__ORCL_ret_1d", "ROST_RossStores_ret_1d__zrel__ORCL_ret_1d", "heston_xi__minus__ADBE_vol_20d", "IBEX_Spain_ret_20d__div__ROST_RossStores_zscore_60d", "ES_Evergy_ret_1d"], "is_new": false}, {"model_id": "v1_h1_NORMAL_LogisticRegression_N19", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 1, "n_features": 19, "F1_dir": 0.5377, "F1_UP_FORT": 0.2544, "F1_DOWN_FORT": 0.3673, "train_start": "2000-11-16", "sampler": "SMOTETomek", "best_params": "{}", "features": ["heston_xi__minus__COF_CapitalOne_ret_5d", "XLY_Disc_zscore_60d__prod__ADM_ArcherDaniels_ret_1d", "US1Y_Rate_ret_20d", "heston_xi__minus__EQIX_Equinix_vol_20d", "VRP", "heston_xi__div__ASML_ASML_vol_20d", "vix_zscore_10d", "IBEX_Spain_ret_20d__macross__ROST_RossStores_zscore_60d", "ROST_RossStores_ret_1d__ret5x__EQIX_Equinix_vol_20d", "ROST_RossStores_ret_1d__macross__WMT_ret_20d", "ADM_ArcherDaniels_ret_1d__ret5x__ADBE_vol_20d", "NFCI_ret_5d__minus__ORCL_ret_1d", "MO_AltriaMG_ret_1d", "NFCI_ret_5d__zrel__ORCL_ret_1d", "ROST_RossStores_ret_1d__zrel__ORCL_ret_1d", "heston_xi__minus__ADBE_vol_20d", "IBEX_Spain_ret_20d__div__ROST_RossStores_zscore_60d", "ES_Evergy_ret_1d", "MSTR_Bitcoin3_ret_1d"], "is_new": false}, {"model_id": "v1_h1_NORMAL_LogisticRegression_N20", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 1, "n_features": 20, "F1_dir": 0.5362, "F1_UP_FORT": 0.2356, "F1_DOWN_FORT": 0.3781, "train_start": "2000-11-16", "sampler": "SMOTETomek", "best_params": "{}", "features": ["heston_xi__minus__COF_CapitalOne_ret_5d", "XLY_Disc_zscore_60d__prod__ADM_ArcherDaniels_ret_1d", "US1Y_Rate_ret_20d", "heston_xi__minus__EQIX_Equinix_vol_20d", "VRP", "heston_xi__div__ASML_ASML_vol_20d", "vix_zscore_10d", "IBEX_Spain_ret_20d__macross__ROST_RossStores_zscore_60d", "ROST_RossStores_ret_1d__ret5x__EQIX_Equinix_vol_20d", "ROST_RossStores_ret_1d__macross__WMT_ret_20d", "ADM_ArcherDaniels_ret_1d__ret5x__ADBE_vol_20d", "NFCI_ret_5d__minus__ORCL_ret_1d", "MO_AltriaMG_ret_1d", "NFCI_ret_5d__zrel__ORCL_ret_1d", "ROST_RossStores_ret_1d__zrel__ORCL_ret_1d", "heston_xi__minus__ADBE_vol_20d", "IBEX_Spain_ret_20d__div__ROST_RossStores_zscore_60d", "ES_Evergy_ret_1d", "MSTR_Bitcoin3_ret_1d", "LMT_LockheedMartin_ret_1d"], "is_new": false}, {"model_id": "v1_h1_NORMAL_LogisticRegression_N21", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 1, "n_features": 21, "F1_dir": 0.5023, "F1_UP_FORT": 0.2167, "F1_DOWN_FORT": 0.3675, "train_start": "2000-11-16", "sampler": "SMOTETomek", "best_params": "{}", "features": ["heston_xi__minus__COF_CapitalOne_ret_5d", "XLY_Disc_zscore_60d__prod__ADM_ArcherDaniels_ret_1d", "US1Y_Rate_ret_20d", "heston_xi__minus__EQIX_Equinix_vol_20d", "VRP", "heston_xi__div__ASML_ASML_vol_20d", "vix_zscore_10d", "IBEX_Spain_ret_20d__macross__ROST_RossStores_zscore_60d", "ROST_RossStores_ret_1d__ret5x__EQIX_Equinix_vol_20d", "ROST_RossStores_ret_1d__macross__WMT_ret_20d", "ADM_ArcherDaniels_ret_1d__ret5x__ADBE_vol_20d", "NFCI_ret_5d__minus__ORCL_ret_1d", "MO_AltriaMG_ret_1d", "NFCI_ret_5d__zrel__ORCL_ret_1d", "ROST_RossStores_ret_1d__zrel__ORCL_ret_1d", "heston_xi__minus__ADBE_vol_20d", "IBEX_Spain_ret_20d__div__ROST_RossStores_zscore_60d", "ES_Evergy_ret_1d", "MSTR_Bitcoin3_ret_1d", "LMT_LockheedMartin_ret_1d", "INTC_ret_1d"], "is_new": false}, {"model_id": "v1_h1_NORMAL_LogisticRegression_N22", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 1, "n_features": 22, "F1_dir": 0.5392, "F1_UP_FORT": 0.2515, "F1_DOWN_FORT": 0.3558, "train_start": "2000-11-16", "sampler": "SMOTETomek", "best_params": "{}", "features": ["heston_xi__minus__COF_CapitalOne_ret_5d", "XLY_Disc_zscore_60d__prod__ADM_ArcherDaniels_ret_1d", "US1Y_Rate_ret_20d", "heston_xi__minus__EQIX_Equinix_vol_20d", "VRP", "heston_xi__div__ASML_ASML_vol_20d", "vix_zscore_10d", "IBEX_Spain_ret_20d__macross__ROST_RossStores_zscore_60d", "ROST_RossStores_ret_1d__ret5x__EQIX_Equinix_vol_20d", "ROST_RossStores_ret_1d__macross__WMT_ret_20d", "ADM_ArcherDaniels_ret_1d__ret5x__ADBE_vol_20d", "NFCI_ret_5d__minus__ORCL_ret_1d", "MO_AltriaMG_ret_1d", "NFCI_ret_5d__zrel__ORCL_ret_1d", "ROST_RossStores_ret_1d__zrel__ORCL_ret_1d", "heston_xi__minus__ADBE_vol_20d", "IBEX_Spain_ret_20d__div__ROST_RossStores_zscore_60d", "ES_Evergy_ret_1d", "MSTR_Bitcoin3_ret_1d", "LMT_LockheedMartin_ret_1d", "INTC_ret_1d", "VIX_Price_zscore_60d__minus__ROST_RossStores_zscore_60d"], "is_new": false}, {"model_id": "v1_h1_NORMAL_LogisticRegression_N23", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 1, "n_features": 23, "F1_dir": 0.5367, "F1_UP_FORT": 0.2706, "F1_DOWN_FORT": 0.3352, "train_start": "2000-11-16", "sampler": "SMOTETomek", "best_params": "{}", "features": ["heston_xi__minus__COF_CapitalOne_ret_5d", "XLY_Disc_zscore_60d__prod__ADM_ArcherDaniels_ret_1d", "US1Y_Rate_ret_20d", "heston_xi__minus__EQIX_Equinix_vol_20d", "VRP", "heston_xi__div__ASML_ASML_vol_20d", "vix_zscore_10d", "IBEX_Spain_ret_20d__macross__ROST_RossStores_zscore_60d", "ROST_RossStores_ret_1d__ret5x__EQIX_Equinix_vol_20d", "ROST_RossStores_ret_1d__macross__WMT_ret_20d", "ADM_ArcherDaniels_ret_1d__ret5x__ADBE_vol_20d", "NFCI_ret_5d__minus__ORCL_ret_1d", "MO_AltriaMG_ret_1d", "NFCI_ret_5d__zrel__ORCL_ret_1d", "ROST_RossStores_ret_1d__zrel__ORCL_ret_1d", "heston_xi__minus__ADBE_vol_20d", "IBEX_Spain_ret_20d__div__ROST_RossStores_zscore_60d", "ES_Evergy_ret_1d", "MSTR_Bitcoin3_ret_1d", "LMT_LockheedMartin_ret_1d", "INTC_ret_1d", "VIX_Price_zscore_60d__minus__ROST_RossStores_zscore_60d", "VIX_Price_zscore_60d__prod__WMT_ret_20d"], "is_new": false}, {"model_id": "v1_h1_NORMAL_LogisticRegression_N24", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 1, "n_features": 24, "F1_dir": 0.5318, "F1_UP_FORT": 0.2448, "F1_DOWN_FORT": 0.3444, "train_start": "2000-11-16", "sampler": "SMOTETomek", "best_params": "{}", "features": ["heston_xi__minus__COF_CapitalOne_ret_5d", "XLY_Disc_zscore_60d__prod__ADM_ArcherDaniels_ret_1d", "US1Y_Rate_ret_20d", "heston_xi__minus__EQIX_Equinix_vol_20d", "VRP", "heston_xi__div__ASML_ASML_vol_20d", "vix_zscore_10d", "IBEX_Spain_ret_20d__macross__ROST_RossStores_zscore_60d", "ROST_RossStores_ret_1d__ret5x__EQIX_Equinix_vol_20d", "ROST_RossStores_ret_1d__macross__WMT_ret_20d", "ADM_ArcherDaniels_ret_1d__ret5x__ADBE_vol_20d", "NFCI_ret_5d__minus__ORCL_ret_1d", "MO_AltriaMG_ret_1d", "NFCI_ret_5d__zrel__ORCL_ret_1d", "ROST_RossStores_ret_1d__zrel__ORCL_ret_1d", "heston_xi__minus__ADBE_vol_20d", "IBEX_Spain_ret_20d__div__ROST_RossStores_zscore_60d", "ES_Evergy_ret_1d", "MSTR_Bitcoin3_ret_1d", "LMT_LockheedMartin_ret_1d", "INTC_ret_1d", "VIX_Price_zscore_60d__minus__ROST_RossStores_zscore_60d", "VIX_Price_zscore_60d__prod__WMT_ret_20d", "heston_xi__zrel__XLY_Disc_zscore_60d"], "is_new": false}, {"model_id": "v1_h1_NORMAL_LogisticRegression_N25", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 1, "n_features": 25, "F1_dir": 0.5089, "F1_UP_FORT": 0.2065, "F1_DOWN_FORT": 0.349, "train_start": "2000-11-16", "sampler": "SMOTETomek", "best_params": "{}", "features": ["heston_xi__minus__COF_CapitalOne_ret_5d", "XLY_Disc_zscore_60d__prod__ADM_ArcherDaniels_ret_1d", "US1Y_Rate_ret_20d", "heston_xi__minus__EQIX_Equinix_vol_20d", "VRP", "heston_xi__div__ASML_ASML_vol_20d", "vix_zscore_10d", "IBEX_Spain_ret_20d__macross__ROST_RossStores_zscore_60d", "ROST_RossStores_ret_1d__ret5x__EQIX_Equinix_vol_20d", "ROST_RossStores_ret_1d__macross__WMT_ret_20d", "ADM_ArcherDaniels_ret_1d__ret5x__ADBE_vol_20d", "NFCI_ret_5d__minus__ORCL_ret_1d", "MO_AltriaMG_ret_1d", "NFCI_ret_5d__zrel__ORCL_ret_1d", "ROST_RossStores_ret_1d__zrel__ORCL_ret_1d", "heston_xi__minus__ADBE_vol_20d", "IBEX_Spain_ret_20d__div__ROST_RossStores_zscore_60d", "ES_Evergy_ret_1d", "MSTR_Bitcoin3_ret_1d", "LMT_LockheedMartin_ret_1d", "INTC_ret_1d", "VIX_Price_zscore_60d__minus__ROST_RossStores_zscore_60d", "VIX_Price_zscore_60d__prod__WMT_ret_20d", "heston_xi__zrel__XLY_Disc_zscore_60d", "YUM_YumBrands_ret_1d__minus__WMT_ret_20d"], "is_new": false}, {"model_id": "v1_h1_NORMAL_LightGBM_Optuna_N15", "algo": "LightGBM", "regime": "NORMAL", "horizon": 1, "n_features": 15, "F1_dir": 0.5403, "F1_UP_FORT": 0.2914, "F1_DOWN_FORT": 0.3272, "train_start": "2000-11-16", "sampler": "SMOTETomek", "best_params": "{}", "features": ["heston_xi__minus__COF_CapitalOne_ret_5d", "XLY_Disc_zscore_60d__prod__ADM_ArcherDaniels_ret_1d", "US1Y_Rate_ret_20d", "heston_xi__minus__EQIX_Equinix_vol_20d", "VRP", "heston_xi__div__ASML_ASML_vol_20d", "vix_zscore_10d", "IBEX_Spain_ret_20d__macross__ROST_RossStores_zscore_60d", "ROST_RossStores_ret_1d__ret5x__EQIX_Equinix_vol_20d", "ROST_RossStores_ret_1d__macross__WMT_ret_20d", "ADM_ArcherDaniels_ret_1d__ret5x__ADBE_vol_20d", "NFCI_ret_5d__minus__ORCL_ret_1d", "MO_AltriaMG_ret_1d", "NFCI_ret_5d__zrel__ORCL_ret_1d", "ROST_RossStores_ret_1d__zrel__ORCL_ret_1d"], "is_new": false}, {"model_id": "v1_h1_NORMAL_LightGBM_OptunaCal_N15", "algo": "LightGBMCal", "regime": "NORMAL", "horizon": 1, "n_features": 15, "F1_dir": 0.5303, "F1_UP_FORT": 0.2797, "F1_DOWN_FORT": 0.1897, "train_start": "2000-11-16", "sampler": "SMOTETomek", "best_params": "{}", "features": ["heston_xi__minus__COF_CapitalOne_ret_5d", "XLY_Disc_zscore_60d__prod__ADM_ArcherDaniels_ret_1d", "US1Y_Rate_ret_20d", "heston_xi__minus__EQIX_Equinix_vol_20d", "VRP", "heston_xi__div__ASML_ASML_vol_20d", "vix_zscore_10d", "IBEX_Spain_ret_20d__macross__ROST_RossStores_zscore_60d", "ROST_RossStores_ret_1d__ret5x__EQIX_Equinix_vol_20d", "ROST_RossStores_ret_1d__macross__WMT_ret_20d", "ADM_ArcherDaniels_ret_1d__ret5x__ADBE_vol_20d", "NFCI_ret_5d__minus__ORCL_ret_1d", "MO_AltriaMG_ret_1d", "NFCI_ret_5d__zrel__ORCL_ret_1d", "ROST_RossStores_ret_1d__zrel__ORCL_ret_1d"], "is_new": false}, {"model_id": "v1_h1_STRESS_XGBoost_N5", "algo": "XGBoost", "regime": "STRESS", "horizon": 1, "n_features": 5, "F1_dir": 0.5742, "F1_UP_FORT": 0.2771, "F1_DOWN_FORT": 0.2967, "train_start": "2001-02-08", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["ENB_EnbridgeInc_ret_1d__macross__TXN_ret_20d", "ETN_Eaton_ret_1d__div__US6M_Rate_ret_1d", "egarch_spx_delta_h1__macross__VRTX_VertexPharm_ret_1d", "egarch_spx_delta_h1__div__LMT_LockheedMartin_ret_1d", "EXC_Exelon_zscore_60d"], "is_new": false}, {"model_id": "v1_h1_STRESS_XGBoost_N6", "algo": "XGBoost", "regime": "STRESS", "horizon": 1, "n_features": 6, "F1_dir": 0.5411, "F1_UP_FORT": 0.3312, "F1_DOWN_FORT": 0.4425, "train_start": "2001-02-08", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["ENB_EnbridgeInc_ret_1d__macross__TXN_ret_20d", "ETN_Eaton_ret_1d__div__US6M_Rate_ret_1d", "egarch_spx_delta_h1__macross__VRTX_VertexPharm_ret_1d", "egarch_spx_delta_h1__div__LMT_LockheedMartin_ret_1d", "EXC_Exelon_zscore_60d", "vix_mean_abs_ret_5d"], "is_new": false}, {"model_id": "v1_h1_STRESS_XGBoost_N7", "algo": "XGBoost", "regime": "STRESS", "horizon": 1, "n_features": 7, "F1_dir": 0.5607, "F1_UP_FORT": 0.2987, "F1_DOWN_FORT": 0.4344, "train_start": "2001-02-08", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["ENB_EnbridgeInc_ret_1d__macross__TXN_ret_20d", "ETN_Eaton_ret_1d__div__US6M_Rate_ret_1d", "egarch_spx_delta_h1__macross__VRTX_VertexPharm_ret_1d", "egarch_spx_delta_h1__div__LMT_LockheedMartin_ret_1d", "EXC_Exelon_zscore_60d", "vix_mean_abs_ret_5d", "LMT_LockheedMartin_ret_1d__macross__MCD_ret_1d"], "is_new": false}, {"model_id": "v1_h1_STRESS_XGBoost_N8", "algo": "XGBoost", "regime": "STRESS", "horizon": 1, "n_features": 8, "F1_dir": 0.5839, "F1_UP_FORT": 0.321, "F1_DOWN_FORT": 0.466, "train_start": "2001-02-08", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["ENB_EnbridgeInc_ret_1d__macross__TXN_ret_20d", "ETN_Eaton_ret_1d__div__US6M_Rate_ret_1d", "egarch_spx_delta_h1__macross__VRTX_VertexPharm_ret_1d", "egarch_spx_delta_h1__div__LMT_LockheedMartin_ret_1d", "EXC_Exelon_zscore_60d", "vix_mean_abs_ret_5d", "LMT_LockheedMartin_ret_1d__macross__MCD_ret_1d", "BLK_BlackRock_zscore_60d__ret5x__AMD_ret_5d"], "is_new": false}, {"model_id": "v1_h1_STRESS_XGBoost_N9", "algo": "XGBoost", "regime": "STRESS", "horizon": 1, "n_features": 9, "F1_dir": 0.5458, "F1_UP_FORT": 0.3125, "F1_DOWN_FORT": 0.455, "train_start": "2001-02-08", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["ENB_EnbridgeInc_ret_1d__macross__TXN_ret_20d", "ETN_Eaton_ret_1d__div__US6M_Rate_ret_1d", "egarch_spx_delta_h1__macross__VRTX_VertexPharm_ret_1d", "egarch_spx_delta_h1__div__LMT_LockheedMartin_ret_1d", "EXC_Exelon_zscore_60d", "vix_mean_abs_ret_5d", "LMT_LockheedMartin_ret_1d__macross__MCD_ret_1d", "BLK_BlackRock_zscore_60d__ret5x__AMD_ret_5d", "VIX_Price_ret_5d__ret5x__LMT_LockheedMartin_ret_1d"], "is_new": false}, {"model_id": "v1_h1_STRESS_XGBoost_N10", "algo": "XGBoost", "regime": "STRESS", "horizon": 1, "n_features": 10, "F1_dir": 0.5361, "F1_UP_FORT": 0.3077, "F1_DOWN_FORT": 0.5145, "train_start": "2001-02-08", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["ENB_EnbridgeInc_ret_1d__macross__TXN_ret_20d", "ETN_Eaton_ret_1d__div__US6M_Rate_ret_1d", "egarch_spx_delta_h1__macross__VRTX_VertexPharm_ret_1d", "egarch_spx_delta_h1__div__LMT_LockheedMartin_ret_1d", "EXC_Exelon_zscore_60d", "vix_mean_abs_ret_5d", "LMT_LockheedMartin_ret_1d__macross__MCD_ret_1d", "BLK_BlackRock_zscore_60d__ret5x__AMD_ret_5d", "VIX_Price_ret_5d__ret5x__LMT_LockheedMartin_ret_1d", "heston_var_ev_h7"], "is_new": false}, {"model_id": "v1_h1_STRESS_XGBoost_N11", "algo": "XGBoost", "regime": "STRESS", "horizon": 1, "n_features": 11, "F1_dir": 0.5533, "F1_UP_FORT": 0.3253, "F1_DOWN_FORT": 0.5106, "train_start": "2001-02-08", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["ENB_EnbridgeInc_ret_1d__macross__TXN_ret_20d", "ETN_Eaton_ret_1d__div__US6M_Rate_ret_1d", "egarch_spx_delta_h1__macross__VRTX_VertexPharm_ret_1d", "egarch_spx_delta_h1__div__LMT_LockheedMartin_ret_1d", "EXC_Exelon_zscore_60d", "vix_mean_abs_ret_5d", "LMT_LockheedMartin_ret_1d__macross__MCD_ret_1d", "BLK_BlackRock_zscore_60d__ret5x__AMD_ret_5d", "VIX_Price_ret_5d__ret5x__LMT_LockheedMartin_ret_1d", "heston_var_ev_h7", "VIX_Price_ret_5d__prod__LMT_LockheedMartin_ret_1d"], "is_new": false}, {"model_id": "v1_h1_STRESS_XGBoost_N12", "algo": "XGBoost", "regime": "STRESS", "horizon": 1, "n_features": 12, "F1_dir": 0.5508, "F1_UP_FORT": 0.319, "F1_DOWN_FORT": 0.4874, "train_start": "2001-02-08", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["ENB_EnbridgeInc_ret_1d__macross__TXN_ret_20d", "ETN_Eaton_ret_1d__div__US6M_Rate_ret_1d", "egarch_spx_delta_h1__macross__VRTX_VertexPharm_ret_1d", "egarch_spx_delta_h1__div__LMT_LockheedMartin_ret_1d", "EXC_Exelon_zscore_60d", "vix_mean_abs_ret_5d", "LMT_LockheedMartin_ret_1d__macross__MCD_ret_1d", "BLK_BlackRock_zscore_60d__ret5x__AMD_ret_5d", "VIX_Price_ret_5d__ret5x__LMT_LockheedMartin_ret_1d", "heston_var_ev_h7", "VIX_Price_ret_5d__prod__LMT_LockheedMartin_ret_1d", "ETN_Eaton_ret_1d__macross__LMT_LockheedMartin_ret_1d"], "is_new": false}, {"model_id": "v1_h1_STRESS_XGBoost_N13", "algo": "XGBoost", "regime": "STRESS", "horizon": 1, "n_features": 13, "F1_dir": 0.5655, "F1_UP_FORT": 0.3392, "F1_DOWN_FORT": 0.4737, "train_start": "2001-02-08", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["ENB_EnbridgeInc_ret_1d__macross__TXN_ret_20d", "ETN_Eaton_ret_1d__div__US6M_Rate_ret_1d", "egarch_spx_delta_h1__macross__VRTX_VertexPharm_ret_1d", "egarch_spx_delta_h1__div__LMT_LockheedMartin_ret_1d", "EXC_Exelon_zscore_60d", "vix_mean_abs_ret_5d", "LMT_LockheedMartin_ret_1d__macross__MCD_ret_1d", "BLK_BlackRock_zscore_60d__ret5x__AMD_ret_5d", "VIX_Price_ret_5d__ret5x__LMT_LockheedMartin_ret_1d", "heston_var_ev_h7", "VIX_Price_ret_5d__prod__LMT_LockheedMartin_ret_1d", "ETN_Eaton_ret_1d__macross__LMT_LockheedMartin_ret_1d", "XEL_Xcel_ret_1d__ret5x__DE_Deere_ret_1d"], "is_new": false}, {"model_id": "v1_h1_STRESS_XGBoost_N14", "algo": "XGBoost", "regime": "STRESS", "horizon": 1, "n_features": 14, "F1_dir": 0.5497, "F1_UP_FORT": 0.3077, "F1_DOWN_FORT": 0.4746, "train_start": "2001-02-08", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["ENB_EnbridgeInc_ret_1d__macross__TXN_ret_20d", "ETN_Eaton_ret_1d__div__US6M_Rate_ret_1d", "egarch_spx_delta_h1__macross__VRTX_VertexPharm_ret_1d", "egarch_spx_delta_h1__div__LMT_LockheedMartin_ret_1d", "EXC_Exelon_zscore_60d", "vix_mean_abs_ret_5d", "LMT_LockheedMartin_ret_1d__macross__MCD_ret_1d", "BLK_BlackRock_zscore_60d__ret5x__AMD_ret_5d", "VIX_Price_ret_5d__ret5x__LMT_LockheedMartin_ret_1d", "heston_var_ev_h7", "VIX_Price_ret_5d__prod__LMT_LockheedMartin_ret_1d", "ETN_Eaton_ret_1d__macross__LMT_LockheedMartin_ret_1d", "XEL_Xcel_ret_1d__ret5x__DE_Deere_ret_1d", "CMCSA_ret_1d"], "is_new": false}, {"model_id": "v1_h1_STRESS_XGBoost_N15", "algo": "XGBoost", "regime": "STRESS", "horizon": 1, "n_features": 15, "F1_dir": 0.5398, "F1_UP_FORT": 0.3133, "F1_DOWN_FORT": 0.4681, "train_start": "2001-02-08", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["ENB_EnbridgeInc_ret_1d__macross__TXN_ret_20d", "ETN_Eaton_ret_1d__div__US6M_Rate_ret_1d", "egarch_spx_delta_h1__macross__VRTX_VertexPharm_ret_1d", "egarch_spx_delta_h1__div__LMT_LockheedMartin_ret_1d", "EXC_Exelon_zscore_60d", "vix_mean_abs_ret_5d", "LMT_LockheedMartin_ret_1d__macross__MCD_ret_1d", "BLK_BlackRock_zscore_60d__ret5x__AMD_ret_5d", "VIX_Price_ret_5d__ret5x__LMT_LockheedMartin_ret_1d", "heston_var_ev_h7", "VIX_Price_ret_5d__prod__LMT_LockheedMartin_ret_1d", "ETN_Eaton_ret_1d__macross__LMT_LockheedMartin_ret_1d", "XEL_Xcel_ret_1d__ret5x__DE_Deere_ret_1d", "CMCSA_ret_1d", "heston_var_ev_h3"], "is_new": false}, {"model_id": "v1_h1_STRESS_XGBoost_N16", "algo": "XGBoost", "regime": "STRESS", "horizon": 1, "n_features": 16, "F1_dir": 0.5184, "F1_UP_FORT": 0.2857, "F1_DOWN_FORT": 0.4711, "train_start": "2001-02-08", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["ENB_EnbridgeInc_ret_1d__macross__TXN_ret_20d", "ETN_Eaton_ret_1d__div__US6M_Rate_ret_1d", "egarch_spx_delta_h1__macross__VRTX_VertexPharm_ret_1d", "egarch_spx_delta_h1__div__LMT_LockheedMartin_ret_1d", "EXC_Exelon_zscore_60d", "vix_mean_abs_ret_5d", "LMT_LockheedMartin_ret_1d__macross__MCD_ret_1d", "BLK_BlackRock_zscore_60d__ret5x__AMD_ret_5d", "VIX_Price_ret_5d__ret5x__LMT_LockheedMartin_ret_1d", "heston_var_ev_h7", "VIX_Price_ret_5d__prod__LMT_LockheedMartin_ret_1d", "ETN_Eaton_ret_1d__macross__LMT_LockheedMartin_ret_1d", "XEL_Xcel_ret_1d__ret5x__DE_Deere_ret_1d", "CMCSA_ret_1d", "heston_var_ev_h3", "CPB_CampbellSoup_ret_5d"], "is_new": false}, {"model_id": "v1_h1_STRESS_XGBoost_N17", "algo": "XGBoost", "regime": "STRESS", "horizon": 1, "n_features": 17, "F1_dir": 0.5495, "F1_UP_FORT": 0.3106, "F1_DOWN_FORT": 0.498, "train_start": "2001-02-08", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["ENB_EnbridgeInc_ret_1d__macross__TXN_ret_20d", "ETN_Eaton_ret_1d__div__US6M_Rate_ret_1d", "egarch_spx_delta_h1__macross__VRTX_VertexPharm_ret_1d", "egarch_spx_delta_h1__div__LMT_LockheedMartin_ret_1d", "EXC_Exelon_zscore_60d", "vix_mean_abs_ret_5d", "LMT_LockheedMartin_ret_1d__macross__MCD_ret_1d", "BLK_BlackRock_zscore_60d__ret5x__AMD_ret_5d", "VIX_Price_ret_5d__ret5x__LMT_LockheedMartin_ret_1d", "heston_var_ev_h7", "VIX_Price_ret_5d__prod__LMT_LockheedMartin_ret_1d", "ETN_Eaton_ret_1d__macross__LMT_LockheedMartin_ret_1d", "XEL_Xcel_ret_1d__ret5x__DE_Deere_ret_1d", "CMCSA_ret_1d", "heston_var_ev_h3", "CPB_CampbellSoup_ret_5d", "EWJ_Japan_vol_20d"], "is_new": false}, {"model_id": "v1_h1_STRESS_XGBoost_N18", "algo": "XGBoost", "regime": "STRESS", "horizon": 1, "n_features": 18, "F1_dir": 0.5312, "F1_UP_FORT": 0.303, "F1_DOWN_FORT": 0.4737, "train_start": "2001-02-08", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["ENB_EnbridgeInc_ret_1d__macross__TXN_ret_20d", "ETN_Eaton_ret_1d__div__US6M_Rate_ret_1d", "egarch_spx_delta_h1__macross__VRTX_VertexPharm_ret_1d", "egarch_spx_delta_h1__div__LMT_LockheedMartin_ret_1d", "EXC_Exelon_zscore_60d", "vix_mean_abs_ret_5d", "LMT_LockheedMartin_ret_1d__macross__MCD_ret_1d", "BLK_BlackRock_zscore_60d__ret5x__AMD_ret_5d", "VIX_Price_ret_5d__ret5x__LMT_LockheedMartin_ret_1d", "heston_var_ev_h7", "VIX_Price_ret_5d__prod__LMT_LockheedMartin_ret_1d", "ETN_Eaton_ret_1d__macross__LMT_LockheedMartin_ret_1d", "XEL_Xcel_ret_1d__ret5x__DE_Deere_ret_1d", "CMCSA_ret_1d", "heston_var_ev_h3", "CPB_CampbellSoup_ret_5d", "EWJ_Japan_vol_20d", "ENB_EnbridgeInc_ret_1d__ret5x__XEL_Xcel_ret_1d"], "is_new": false}, {"model_id": "v1_h1_STRESS_XGBoost_N19", "algo": "XGBoost", "regime": "STRESS", "horizon": 1, "n_features": 19, "F1_dir": 0.5347, "F1_UP_FORT": 0.2642, "F1_DOWN_FORT": 0.4681, "train_start": "2001-02-08", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["ENB_EnbridgeInc_ret_1d__macross__TXN_ret_20d", "ETN_Eaton_ret_1d__div__US6M_Rate_ret_1d", "egarch_spx_delta_h1__macross__VRTX_VertexPharm_ret_1d", "egarch_spx_delta_h1__div__LMT_LockheedMartin_ret_1d", "EXC_Exelon_zscore_60d", "vix_mean_abs_ret_5d", "LMT_LockheedMartin_ret_1d__macross__MCD_ret_1d", "BLK_BlackRock_zscore_60d__ret5x__AMD_ret_5d", "VIX_Price_ret_5d__ret5x__LMT_LockheedMartin_ret_1d", "heston_var_ev_h7", "VIX_Price_ret_5d__prod__LMT_LockheedMartin_ret_1d", "ETN_Eaton_ret_1d__macross__LMT_LockheedMartin_ret_1d", "XEL_Xcel_ret_1d__ret5x__DE_Deere_ret_1d", "CMCSA_ret_1d", "heston_var_ev_h3", "CPB_CampbellSoup_ret_5d", "EWJ_Japan_vol_20d", "ENB_EnbridgeInc_ret_1d__ret5x__XEL_Xcel_ret_1d", "vix_vs_ma20__ret5x__VRTX_VertexPharm_ret_1d"], "is_new": false}, {"model_id": "v1_h1_STRESS_XGBoost_N20", "algo": "XGBoost", "regime": "STRESS", "horizon": 1, "n_features": 20, "F1_dir": 0.5366, "F1_UP_FORT": 0.2763, "F1_DOWN_FORT": 0.4245, "train_start": "2001-02-08", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["ENB_EnbridgeInc_ret_1d__macross__TXN_ret_20d", "ETN_Eaton_ret_1d__div__US6M_Rate_ret_1d", "egarch_spx_delta_h1__macross__VRTX_VertexPharm_ret_1d", "egarch_spx_delta_h1__div__LMT_LockheedMartin_ret_1d", "EXC_Exelon_zscore_60d", "vix_mean_abs_ret_5d", "LMT_LockheedMartin_ret_1d__macross__MCD_ret_1d", "BLK_BlackRock_zscore_60d__ret5x__AMD_ret_5d", "VIX_Price_ret_5d__ret5x__LMT_LockheedMartin_ret_1d", "heston_var_ev_h7", "VIX_Price_ret_5d__prod__LMT_LockheedMartin_ret_1d", "ETN_Eaton_ret_1d__macross__LMT_LockheedMartin_ret_1d", "XEL_Xcel_ret_1d__ret5x__DE_Deere_ret_1d", "CMCSA_ret_1d", "heston_var_ev_h3", "CPB_CampbellSoup_ret_5d", "EWJ_Japan_vol_20d", "ENB_EnbridgeInc_ret_1d__ret5x__XEL_Xcel_ret_1d", "vix_vs_ma20__ret5x__VRTX_VertexPharm_ret_1d", "PPL_PPL_ret_1d__div__BLK_BlackRock_zscore_60d"], "is_new": false}, {"model_id": "v1_h1_STRESS_XGBoost_N21", "algo": "XGBoost", "regime": "STRESS", "horizon": 1, "n_features": 21, "F1_dir": 0.5318, "F1_UP_FORT": 0.2577, "F1_DOWN_FORT": 0.4715, "train_start": "2001-02-08", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["ENB_EnbridgeInc_ret_1d__macross__TXN_ret_20d", "ETN_Eaton_ret_1d__div__US6M_Rate_ret_1d", "egarch_spx_delta_h1__macross__VRTX_VertexPharm_ret_1d", "egarch_spx_delta_h1__div__LMT_LockheedMartin_ret_1d", "EXC_Exelon_zscore_60d", "vix_mean_abs_ret_5d", "LMT_LockheedMartin_ret_1d__macross__MCD_ret_1d", "BLK_BlackRock_zscore_60d__ret5x__AMD_ret_5d", "VIX_Price_ret_5d__ret5x__LMT_LockheedMartin_ret_1d", "heston_var_ev_h7", "VIX_Price_ret_5d__prod__LMT_LockheedMartin_ret_1d", "ETN_Eaton_ret_1d__macross__LMT_LockheedMartin_ret_1d", "XEL_Xcel_ret_1d__ret5x__DE_Deere_ret_1d", "CMCSA_ret_1d", "heston_var_ev_h3", "CPB_CampbellSoup_ret_5d", "EWJ_Japan_vol_20d", "ENB_EnbridgeInc_ret_1d__ret5x__XEL_Xcel_ret_1d", "vix_vs_ma20__ret5x__VRTX_VertexPharm_ret_1d", "PPL_PPL_ret_1d__div__BLK_BlackRock_zscore_60d", "NFCI_zscore_60d__macross__VRTX_VertexPharm_ret_1d"], "is_new": false}, {"model_id": "v1_h1_STRESS_XGBoost_N22", "algo": "XGBoost", "regime": "STRESS", "horizon": 1, "n_features": 22, "F1_dir": 0.5026, "F1_UP_FORT": 0.2532, "F1_DOWN_FORT": 0.4861, "train_start": "2001-02-08", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["ENB_EnbridgeInc_ret_1d__macross__TXN_ret_20d", "ETN_Eaton_ret_1d__div__US6M_Rate_ret_1d", "egarch_spx_delta_h1__macross__VRTX_VertexPharm_ret_1d", "egarch_spx_delta_h1__div__LMT_LockheedMartin_ret_1d", "EXC_Exelon_zscore_60d", "vix_mean_abs_ret_5d", "LMT_LockheedMartin_ret_1d__macross__MCD_ret_1d", "BLK_BlackRock_zscore_60d__ret5x__AMD_ret_5d", "VIX_Price_ret_5d__ret5x__LMT_LockheedMartin_ret_1d", "heston_var_ev_h7", "VIX_Price_ret_5d__prod__LMT_LockheedMartin_ret_1d", "ETN_Eaton_ret_1d__macross__LMT_LockheedMartin_ret_1d", "XEL_Xcel_ret_1d__ret5x__DE_Deere_ret_1d", "CMCSA_ret_1d", "heston_var_ev_h3", "CPB_CampbellSoup_ret_5d", "EWJ_Japan_vol_20d", "ENB_EnbridgeInc_ret_1d__ret5x__XEL_Xcel_ret_1d", "vix_vs_ma20__ret5x__VRTX_VertexPharm_ret_1d", "PPL_PPL_ret_1d__div__BLK_BlackRock_zscore_60d", "NFCI_zscore_60d__macross__VRTX_VertexPharm_ret_1d", "PCAR_PaccarInc_ret_5d"], "is_new": false}, {"model_id": "v1_h1_STRESS_LightGBM_N5", "algo": "LightGBM", "regime": "STRESS", "horizon": 1, "n_features": 5, "F1_dir": 0.5408, "F1_UP_FORT": 0.3121, "F1_DOWN_FORT": 0.2651, "train_start": "2001-02-08", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["ENB_EnbridgeInc_ret_1d__macross__TXN_ret_20d", "ETN_Eaton_ret_1d__div__US6M_Rate_ret_1d", "egarch_spx_delta_h1__macross__VRTX_VertexPharm_ret_1d", "egarch_spx_delta_h1__div__LMT_LockheedMartin_ret_1d", "EXC_Exelon_zscore_60d"], "is_new": false}, {"model_id": "v1_h1_STRESS_LightGBM_N6", "algo": "LightGBM", "regime": "STRESS", "horizon": 1, "n_features": 6, "F1_dir": 0.5251, "F1_UP_FORT": 0.3436, "F1_DOWN_FORT": 0.4182, "train_start": "2001-02-08", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["ENB_EnbridgeInc_ret_1d__macross__TXN_ret_20d", "ETN_Eaton_ret_1d__div__US6M_Rate_ret_1d", "egarch_spx_delta_h1__macross__VRTX_VertexPharm_ret_1d", "egarch_spx_delta_h1__div__LMT_LockheedMartin_ret_1d", "EXC_Exelon_zscore_60d", "vix_mean_abs_ret_5d"], "is_new": false}, {"model_id": "v1_h1_STRESS_LightGBM_N7", "algo": "LightGBM", "regime": "STRESS", "horizon": 1, "n_features": 7, "F1_dir": 0.5422, "F1_UP_FORT": 0.2953, "F1_DOWN_FORT": 0.4292, "train_start": "2001-02-08", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["ENB_EnbridgeInc_ret_1d__macross__TXN_ret_20d", "ETN_Eaton_ret_1d__div__US6M_Rate_ret_1d", "egarch_spx_delta_h1__macross__VRTX_VertexPharm_ret_1d", "egarch_spx_delta_h1__div__LMT_LockheedMartin_ret_1d", "EXC_Exelon_zscore_60d", "vix_mean_abs_ret_5d", "LMT_LockheedMartin_ret_1d__macross__MCD_ret_1d"], "is_new": false}, {"model_id": "v1_h1_STRESS_LightGBM_N8", "algo": "LightGBM", "regime": "STRESS", "horizon": 1, "n_features": 8, "F1_dir": 0.5472, "F1_UP_FORT": 0.3014, "F1_DOWN_FORT": 0.4516, "train_start": "2001-02-08", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["ENB_EnbridgeInc_ret_1d__macross__TXN_ret_20d", "ETN_Eaton_ret_1d__div__US6M_Rate_ret_1d", "egarch_spx_delta_h1__macross__VRTX_VertexPharm_ret_1d", "egarch_spx_delta_h1__div__LMT_LockheedMartin_ret_1d", "EXC_Exelon_zscore_60d", "vix_mean_abs_ret_5d", "LMT_LockheedMartin_ret_1d__macross__MCD_ret_1d", "BLK_BlackRock_zscore_60d__ret5x__AMD_ret_5d"], "is_new": false}, {"model_id": "v1_h1_STRESS_LightGBM_N9", "algo": "LightGBM", "regime": "STRESS", "horizon": 1, "n_features": 9, "F1_dir": 0.5129, "F1_UP_FORT": 0.2892, "F1_DOWN_FORT": 0.434, "train_start": "2001-02-08", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["ENB_EnbridgeInc_ret_1d__macross__TXN_ret_20d", "ETN_Eaton_ret_1d__div__US6M_Rate_ret_1d", "egarch_spx_delta_h1__macross__VRTX_VertexPharm_ret_1d", "egarch_spx_delta_h1__div__LMT_LockheedMartin_ret_1d", "EXC_Exelon_zscore_60d", "vix_mean_abs_ret_5d", "LMT_LockheedMartin_ret_1d__macross__MCD_ret_1d", "BLK_BlackRock_zscore_60d__ret5x__AMD_ret_5d", "VIX_Price_ret_5d__ret5x__LMT_LockheedMartin_ret_1d"], "is_new": false}, {"model_id": "v1_h1_STRESS_LightGBM_N10", "algo": "LightGBM", "regime": "STRESS", "horizon": 1, "n_features": 10, "F1_dir": 0.5068, "F1_UP_FORT": 0.2892, "F1_DOWN_FORT": 0.4649, "train_start": "2001-02-08", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["ENB_EnbridgeInc_ret_1d__macross__TXN_ret_20d", "ETN_Eaton_ret_1d__div__US6M_Rate_ret_1d", "egarch_spx_delta_h1__macross__VRTX_VertexPharm_ret_1d", "egarch_spx_delta_h1__div__LMT_LockheedMartin_ret_1d", "EXC_Exelon_zscore_60d", "vix_mean_abs_ret_5d", "LMT_LockheedMartin_ret_1d__macross__MCD_ret_1d", "BLK_BlackRock_zscore_60d__ret5x__AMD_ret_5d", "VIX_Price_ret_5d__ret5x__LMT_LockheedMartin_ret_1d", "heston_var_ev_h7"], "is_new": false}, {"model_id": "v1_h1_STRESS_LightGBM_N11", "algo": "LightGBM", "regime": "STRESS", "horizon": 1, "n_features": 11, "F1_dir": 0.5129, "F1_UP_FORT": 0.3095, "F1_DOWN_FORT": 0.4741, "train_start": "2001-02-08", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["ENB_EnbridgeInc_ret_1d__macross__TXN_ret_20d", "ETN_Eaton_ret_1d__div__US6M_Rate_ret_1d", "egarch_spx_delta_h1__macross__VRTX_VertexPharm_ret_1d", "egarch_spx_delta_h1__div__LMT_LockheedMartin_ret_1d", "EXC_Exelon_zscore_60d", "vix_mean_abs_ret_5d", "LMT_LockheedMartin_ret_1d__macross__MCD_ret_1d", "BLK_BlackRock_zscore_60d__ret5x__AMD_ret_5d", "VIX_Price_ret_5d__ret5x__LMT_LockheedMartin_ret_1d", "heston_var_ev_h7", "VIX_Price_ret_5d__prod__LMT_LockheedMartin_ret_1d"], "is_new": false}, {"model_id": "v1_h1_STRESS_LightGBM_N12", "algo": "LightGBM", "regime": "STRESS", "horizon": 1, "n_features": 12, "F1_dir": 0.5373, "F1_UP_FORT": 0.2767, "F1_DOWN_FORT": 0.4444, "train_start": "2001-02-08", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["ENB_EnbridgeInc_ret_1d__macross__TXN_ret_20d", "ETN_Eaton_ret_1d__div__US6M_Rate_ret_1d", "egarch_spx_delta_h1__macross__VRTX_VertexPharm_ret_1d", "egarch_spx_delta_h1__div__LMT_LockheedMartin_ret_1d", "EXC_Exelon_zscore_60d", "vix_mean_abs_ret_5d", "LMT_LockheedMartin_ret_1d__macross__MCD_ret_1d", "BLK_BlackRock_zscore_60d__ret5x__AMD_ret_5d", "VIX_Price_ret_5d__ret5x__LMT_LockheedMartin_ret_1d", "heston_var_ev_h7", "VIX_Price_ret_5d__prod__LMT_LockheedMartin_ret_1d", "ETN_Eaton_ret_1d__macross__LMT_LockheedMartin_ret_1d"], "is_new": false}, {"model_id": "v1_h1_STRESS_LightGBM_N13", "algo": "LightGBM", "regime": "STRESS", "horizon": 1, "n_features": 13, "F1_dir": 0.536, "F1_UP_FORT": 0.3, "F1_DOWN_FORT": 0.4701, "train_start": "2001-02-08", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["ENB_EnbridgeInc_ret_1d__macross__TXN_ret_20d", "ETN_Eaton_ret_1d__div__US6M_Rate_ret_1d", "egarch_spx_delta_h1__macross__VRTX_VertexPharm_ret_1d", "egarch_spx_delta_h1__div__LMT_LockheedMartin_ret_1d", "EXC_Exelon_zscore_60d", "vix_mean_abs_ret_5d", "LMT_LockheedMartin_ret_1d__macross__MCD_ret_1d", "BLK_BlackRock_zscore_60d__ret5x__AMD_ret_5d", "VIX_Price_ret_5d__ret5x__LMT_LockheedMartin_ret_1d", "heston_var_ev_h7", "VIX_Price_ret_5d__prod__LMT_LockheedMartin_ret_1d", "ETN_Eaton_ret_1d__macross__LMT_LockheedMartin_ret_1d", "XEL_Xcel_ret_1d__ret5x__DE_Deere_ret_1d"], "is_new": false}, {"model_id": "v1_h1_STRESS_LightGBM_N14", "algo": "LightGBM", "regime": "STRESS", "horizon": 1, "n_features": 14, "F1_dir": 0.5481, "F1_UP_FORT": 0.3067, "F1_DOWN_FORT": 0.4473, "train_start": "2001-02-08", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["ENB_EnbridgeInc_ret_1d__macross__TXN_ret_20d", "ETN_Eaton_ret_1d__div__US6M_Rate_ret_1d", "egarch_spx_delta_h1__macross__VRTX_VertexPharm_ret_1d", "egarch_spx_delta_h1__div__LMT_LockheedMartin_ret_1d", "EXC_Exelon_zscore_60d", "vix_mean_abs_ret_5d", "LMT_LockheedMartin_ret_1d__macross__MCD_ret_1d", "BLK_BlackRock_zscore_60d__ret5x__AMD_ret_5d", "VIX_Price_ret_5d__ret5x__LMT_LockheedMartin_ret_1d", "heston_var_ev_h7", "VIX_Price_ret_5d__prod__LMT_LockheedMartin_ret_1d", "ETN_Eaton_ret_1d__macross__LMT_LockheedMartin_ret_1d", "XEL_Xcel_ret_1d__ret5x__DE_Deere_ret_1d", "CMCSA_ret_1d"], "is_new": false}, {"model_id": "v1_h1_STRESS_LightGBM_N15", "algo": "LightGBM", "regime": "STRESS", "horizon": 1, "n_features": 15, "F1_dir": 0.5461, "F1_UP_FORT": 0.3247, "F1_DOWN_FORT": 0.4706, "train_start": "2001-02-08", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["ENB_EnbridgeInc_ret_1d__macross__TXN_ret_20d", "ETN_Eaton_ret_1d__div__US6M_Rate_ret_1d", "egarch_spx_delta_h1__macross__VRTX_VertexPharm_ret_1d", "egarch_spx_delta_h1__div__LMT_LockheedMartin_ret_1d", "EXC_Exelon_zscore_60d", "vix_mean_abs_ret_5d", "LMT_LockheedMartin_ret_1d__macross__MCD_ret_1d", "BLK_BlackRock_zscore_60d__ret5x__AMD_ret_5d", "VIX_Price_ret_5d__ret5x__LMT_LockheedMartin_ret_1d", "heston_var_ev_h7", "VIX_Price_ret_5d__prod__LMT_LockheedMartin_ret_1d", "ETN_Eaton_ret_1d__macross__LMT_LockheedMartin_ret_1d", "XEL_Xcel_ret_1d__ret5x__DE_Deere_ret_1d", "CMCSA_ret_1d", "heston_var_ev_h3"], "is_new": false}, {"model_id": "v1_h1_STRESS_LightGBM_N16", "algo": "LightGBM", "regime": "STRESS", "horizon": 1, "n_features": 16, "F1_dir": 0.5534, "F1_UP_FORT": 0.3273, "F1_DOWN_FORT": 0.4434, "train_start": "2001-02-08", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["ENB_EnbridgeInc_ret_1d__macross__TXN_ret_20d", "ETN_Eaton_ret_1d__div__US6M_Rate_ret_1d", "egarch_spx_delta_h1__macross__VRTX_VertexPharm_ret_1d", "egarch_spx_delta_h1__div__LMT_LockheedMartin_ret_1d", "EXC_Exelon_zscore_60d", "vix_mean_abs_ret_5d", "LMT_LockheedMartin_ret_1d__macross__MCD_ret_1d", "BLK_BlackRock_zscore_60d__ret5x__AMD_ret_5d", "VIX_Price_ret_5d__ret5x__LMT_LockheedMartin_ret_1d", "heston_var_ev_h7", "VIX_Price_ret_5d__prod__LMT_LockheedMartin_ret_1d", "ETN_Eaton_ret_1d__macross__LMT_LockheedMartin_ret_1d", "XEL_Xcel_ret_1d__ret5x__DE_Deere_ret_1d", "CMCSA_ret_1d", "heston_var_ev_h3", "CPB_CampbellSoup_ret_5d"], "is_new": false}, {"model_id": "v1_h1_STRESS_LightGBM_N17", "algo": "LightGBM", "regime": "STRESS", "horizon": 1, "n_features": 17, "F1_dir": 0.5115, "F1_UP_FORT": 0.3145, "F1_DOWN_FORT": 0.431, "train_start": "2001-02-08", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["ENB_EnbridgeInc_ret_1d__macross__TXN_ret_20d", "ETN_Eaton_ret_1d__div__US6M_Rate_ret_1d", "egarch_spx_delta_h1__macross__VRTX_VertexPharm_ret_1d", "egarch_spx_delta_h1__div__LMT_LockheedMartin_ret_1d", "EXC_Exelon_zscore_60d", "vix_mean_abs_ret_5d", "LMT_LockheedMartin_ret_1d__macross__MCD_ret_1d", "BLK_BlackRock_zscore_60d__ret5x__AMD_ret_5d", "VIX_Price_ret_5d__ret5x__LMT_LockheedMartin_ret_1d", "heston_var_ev_h7", "VIX_Price_ret_5d__prod__LMT_LockheedMartin_ret_1d", "ETN_Eaton_ret_1d__macross__LMT_LockheedMartin_ret_1d", "XEL_Xcel_ret_1d__ret5x__DE_Deere_ret_1d", "CMCSA_ret_1d", "heston_var_ev_h3", "CPB_CampbellSoup_ret_5d", "EWJ_Japan_vol_20d"], "is_new": false}, {"model_id": "v1_h1_STRESS_LightGBM_N18", "algo": "LightGBM", "regime": "STRESS", "horizon": 1, "n_features": 18, "F1_dir": 0.5294, "F1_UP_FORT": 0.2857, "F1_DOWN_FORT": 0.4395, "train_start": "2001-02-08", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["ENB_EnbridgeInc_ret_1d__macross__TXN_ret_20d", "ETN_Eaton_ret_1d__div__US6M_Rate_ret_1d", "egarch_spx_delta_h1__macross__VRTX_VertexPharm_ret_1d", "egarch_spx_delta_h1__div__LMT_LockheedMartin_ret_1d", "EXC_Exelon_zscore_60d", "vix_mean_abs_ret_5d", "LMT_LockheedMartin_ret_1d__macross__MCD_ret_1d", "BLK_BlackRock_zscore_60d__ret5x__AMD_ret_5d", "VIX_Price_ret_5d__ret5x__LMT_LockheedMartin_ret_1d", "heston_var_ev_h7", "VIX_Price_ret_5d__prod__LMT_LockheedMartin_ret_1d", "ETN_Eaton_ret_1d__macross__LMT_LockheedMartin_ret_1d", "XEL_Xcel_ret_1d__ret5x__DE_Deere_ret_1d", "CMCSA_ret_1d", "heston_var_ev_h3", "CPB_CampbellSoup_ret_5d", "EWJ_Japan_vol_20d", "ENB_EnbridgeInc_ret_1d__ret5x__XEL_Xcel_ret_1d"], "is_new": false}, {"model_id": "v1_h1_STRESS_LightGBM_N19", "algo": "LightGBM", "regime": "STRESS", "horizon": 1, "n_features": 19, "F1_dir": 0.5532, "F1_UP_FORT": 0.2727, "F1_DOWN_FORT": 0.4726, "train_start": "2001-02-08", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["ENB_EnbridgeInc_ret_1d__macross__TXN_ret_20d", "ETN_Eaton_ret_1d__div__US6M_Rate_ret_1d", "egarch_spx_delta_h1__macross__VRTX_VertexPharm_ret_1d", "egarch_spx_delta_h1__div__LMT_LockheedMartin_ret_1d", "EXC_Exelon_zscore_60d", "vix_mean_abs_ret_5d", "LMT_LockheedMartin_ret_1d__macross__MCD_ret_1d", "BLK_BlackRock_zscore_60d__ret5x__AMD_ret_5d", "VIX_Price_ret_5d__ret5x__LMT_LockheedMartin_ret_1d", "heston_var_ev_h7", "VIX_Price_ret_5d__prod__LMT_LockheedMartin_ret_1d", "ETN_Eaton_ret_1d__macross__LMT_LockheedMartin_ret_1d", "XEL_Xcel_ret_1d__ret5x__DE_Deere_ret_1d", "CMCSA_ret_1d", "heston_var_ev_h3", "CPB_CampbellSoup_ret_5d", "EWJ_Japan_vol_20d", "ENB_EnbridgeInc_ret_1d__ret5x__XEL_Xcel_ret_1d", "vix_vs_ma20__ret5x__VRTX_VertexPharm_ret_1d"], "is_new": false}, {"model_id": "v1_h1_STRESS_LightGBM_N20", "algo": "LightGBM", "regime": "STRESS", "horizon": 1, "n_features": 20, "F1_dir": 0.5366, "F1_UP_FORT": 0.2683, "F1_DOWN_FORT": 0.4192, "train_start": "2001-02-08", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["ENB_EnbridgeInc_ret_1d__macross__TXN_ret_20d", "ETN_Eaton_ret_1d__div__US6M_Rate_ret_1d", "egarch_spx_delta_h1__macross__VRTX_VertexPharm_ret_1d", "egarch_spx_delta_h1__div__LMT_LockheedMartin_ret_1d", "EXC_Exelon_zscore_60d", "vix_mean_abs_ret_5d", "LMT_LockheedMartin_ret_1d__macross__MCD_ret_1d", "BLK_BlackRock_zscore_60d__ret5x__AMD_ret_5d", "VIX_Price_ret_5d__ret5x__LMT_LockheedMartin_ret_1d", "heston_var_ev_h7", "VIX_Price_ret_5d__prod__LMT_LockheedMartin_ret_1d", "ETN_Eaton_ret_1d__macross__LMT_LockheedMartin_ret_1d", "XEL_Xcel_ret_1d__ret5x__DE_Deere_ret_1d", "CMCSA_ret_1d", "heston_var_ev_h3", "CPB_CampbellSoup_ret_5d", "EWJ_Japan_vol_20d", "ENB_EnbridgeInc_ret_1d__ret5x__XEL_Xcel_ret_1d", "vix_vs_ma20__ret5x__VRTX_VertexPharm_ret_1d", "PPL_PPL_ret_1d__div__BLK_BlackRock_zscore_60d"], "is_new": false}, {"model_id": "v1_h1_STRESS_LightGBM_N21", "algo": "LightGBM", "regime": "STRESS", "horizon": 1, "n_features": 21, "F1_dir": 0.5184, "F1_UP_FORT": 0.2743, "F1_DOWN_FORT": 0.4242, "train_start": "2001-02-08", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["ENB_EnbridgeInc_ret_1d__macross__TXN_ret_20d", "ETN_Eaton_ret_1d__div__US6M_Rate_ret_1d", "egarch_spx_delta_h1__macross__VRTX_VertexPharm_ret_1d", "egarch_spx_delta_h1__div__LMT_LockheedMartin_ret_1d", "EXC_Exelon_zscore_60d", "vix_mean_abs_ret_5d", "LMT_LockheedMartin_ret_1d__macross__MCD_ret_1d", "BLK_BlackRock_zscore_60d__ret5x__AMD_ret_5d", "VIX_Price_ret_5d__ret5x__LMT_LockheedMartin_ret_1d", "heston_var_ev_h7", "VIX_Price_ret_5d__prod__LMT_LockheedMartin_ret_1d", "ETN_Eaton_ret_1d__macross__LMT_LockheedMartin_ret_1d", "XEL_Xcel_ret_1d__ret5x__DE_Deere_ret_1d", "CMCSA_ret_1d", "heston_var_ev_h3", "CPB_CampbellSoup_ret_5d", "EWJ_Japan_vol_20d", "ENB_EnbridgeInc_ret_1d__ret5x__XEL_Xcel_ret_1d", "vix_vs_ma20__ret5x__VRTX_VertexPharm_ret_1d", "PPL_PPL_ret_1d__div__BLK_BlackRock_zscore_60d", "NFCI_zscore_60d__macross__VRTX_VertexPharm_ret_1d"], "is_new": false}, {"model_id": "v1_h1_STRESS_LightGBM_N22", "algo": "LightGBM", "regime": "STRESS", "horizon": 1, "n_features": 22, "F1_dir": 0.5246, "F1_UP_FORT": 0.2841, "F1_DOWN_FORT": 0.4159, "train_start": "2001-02-08", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["ENB_EnbridgeInc_ret_1d__macross__TXN_ret_20d", "ETN_Eaton_ret_1d__div__US6M_Rate_ret_1d", "egarch_spx_delta_h1__macross__VRTX_VertexPharm_ret_1d", "egarch_spx_delta_h1__div__LMT_LockheedMartin_ret_1d", "EXC_Exelon_zscore_60d", "vix_mean_abs_ret_5d", "LMT_LockheedMartin_ret_1d__macross__MCD_ret_1d", "BLK_BlackRock_zscore_60d__ret5x__AMD_ret_5d", "VIX_Price_ret_5d__ret5x__LMT_LockheedMartin_ret_1d", "heston_var_ev_h7", "VIX_Price_ret_5d__prod__LMT_LockheedMartin_ret_1d", "ETN_Eaton_ret_1d__macross__LMT_LockheedMartin_ret_1d", "XEL_Xcel_ret_1d__ret5x__DE_Deere_ret_1d", "CMCSA_ret_1d", "heston_var_ev_h3", "CPB_CampbellSoup_ret_5d", "EWJ_Japan_vol_20d", "ENB_EnbridgeInc_ret_1d__ret5x__XEL_Xcel_ret_1d", "vix_vs_ma20__ret5x__VRTX_VertexPharm_ret_1d", "PPL_PPL_ret_1d__div__BLK_BlackRock_zscore_60d", "NFCI_zscore_60d__macross__VRTX_VertexPharm_ret_1d", "PCAR_PaccarInc_ret_5d"], "is_new": false}, {"model_id": "v1_h1_STRESS_GradientBoosting_N5", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 1, "n_features": 5, "F1_dir": 0.5443, "F1_UP_FORT": 0.3164, "F1_DOWN_FORT": 0.2442, "train_start": "2001-02-08", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["ENB_EnbridgeInc_ret_1d__macross__TXN_ret_20d", "ETN_Eaton_ret_1d__div__US6M_Rate_ret_1d", "egarch_spx_delta_h1__macross__VRTX_VertexPharm_ret_1d", "egarch_spx_delta_h1__div__LMT_LockheedMartin_ret_1d", "EXC_Exelon_zscore_60d"], "is_new": false}, {"model_id": "v1_h1_STRESS_GradientBoosting_N6", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 1, "n_features": 6, "F1_dir": 0.5409, "F1_UP_FORT": 0.3515, "F1_DOWN_FORT": 0.3981, "train_start": "2001-02-08", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["ENB_EnbridgeInc_ret_1d__macross__TXN_ret_20d", "ETN_Eaton_ret_1d__div__US6M_Rate_ret_1d", "egarch_spx_delta_h1__macross__VRTX_VertexPharm_ret_1d", "egarch_spx_delta_h1__div__LMT_LockheedMartin_ret_1d", "EXC_Exelon_zscore_60d", "vix_mean_abs_ret_5d"], "is_new": false}, {"model_id": "v1_h1_STRESS_GradientBoosting_N7", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 1, "n_features": 7, "F1_dir": 0.5581, "F1_UP_FORT": 0.3462, "F1_DOWN_FORT": 0.4413, "train_start": "2001-02-08", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["ENB_EnbridgeInc_ret_1d__macross__TXN_ret_20d", "ETN_Eaton_ret_1d__div__US6M_Rate_ret_1d", "egarch_spx_delta_h1__macross__VRTX_VertexPharm_ret_1d", "egarch_spx_delta_h1__div__LMT_LockheedMartin_ret_1d", "EXC_Exelon_zscore_60d", "vix_mean_abs_ret_5d", "LMT_LockheedMartin_ret_1d__macross__MCD_ret_1d"], "is_new": false}, {"model_id": "v1_h1_STRESS_GradientBoosting_N8", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 1, "n_features": 8, "F1_dir": 0.5471, "F1_UP_FORT": 0.3125, "F1_DOWN_FORT": 0.4455, "train_start": "2001-02-08", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["ENB_EnbridgeInc_ret_1d__macross__TXN_ret_20d", "ETN_Eaton_ret_1d__div__US6M_Rate_ret_1d", "egarch_spx_delta_h1__macross__VRTX_VertexPharm_ret_1d", "egarch_spx_delta_h1__div__LMT_LockheedMartin_ret_1d", "EXC_Exelon_zscore_60d", "vix_mean_abs_ret_5d", "LMT_LockheedMartin_ret_1d__macross__MCD_ret_1d", "BLK_BlackRock_zscore_60d__ret5x__AMD_ret_5d"], "is_new": false}, {"model_id": "v1_h1_STRESS_GradientBoosting_N9", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 1, "n_features": 9, "F1_dir": 0.5583, "F1_UP_FORT": 0.2785, "F1_DOWN_FORT": 0.4314, "train_start": "2001-02-08", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["ENB_EnbridgeInc_ret_1d__macross__TXN_ret_20d", "ETN_Eaton_ret_1d__div__US6M_Rate_ret_1d", "egarch_spx_delta_h1__macross__VRTX_VertexPharm_ret_1d", "egarch_spx_delta_h1__div__LMT_LockheedMartin_ret_1d", "EXC_Exelon_zscore_60d", "vix_mean_abs_ret_5d", "LMT_LockheedMartin_ret_1d__macross__MCD_ret_1d", "BLK_BlackRock_zscore_60d__ret5x__AMD_ret_5d", "VIX_Price_ret_5d__ret5x__LMT_LockheedMartin_ret_1d"], "is_new": false}, {"model_id": "v1_h1_STRESS_GradientBoosting_N10", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 1, "n_features": 10, "F1_dir": 0.5519, "F1_UP_FORT": 0.3057, "F1_DOWN_FORT": 0.4828, "train_start": "2001-02-08", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["ENB_EnbridgeInc_ret_1d__macross__TXN_ret_20d", "ETN_Eaton_ret_1d__div__US6M_Rate_ret_1d", "egarch_spx_delta_h1__macross__VRTX_VertexPharm_ret_1d", "egarch_spx_delta_h1__div__LMT_LockheedMartin_ret_1d", "EXC_Exelon_zscore_60d", "vix_mean_abs_ret_5d", "LMT_LockheedMartin_ret_1d__macross__MCD_ret_1d", "BLK_BlackRock_zscore_60d__ret5x__AMD_ret_5d", "VIX_Price_ret_5d__ret5x__LMT_LockheedMartin_ret_1d", "heston_var_ev_h7"], "is_new": false}, {"model_id": "v1_h1_STRESS_GradientBoosting_N11", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 1, "n_features": 11, "F1_dir": 0.5484, "F1_UP_FORT": 0.3106, "F1_DOWN_FORT": 0.4872, "train_start": "2001-02-08", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["ENB_EnbridgeInc_ret_1d__macross__TXN_ret_20d", "ETN_Eaton_ret_1d__div__US6M_Rate_ret_1d", "egarch_spx_delta_h1__macross__VRTX_VertexPharm_ret_1d", "egarch_spx_delta_h1__div__LMT_LockheedMartin_ret_1d", "EXC_Exelon_zscore_60d", "vix_mean_abs_ret_5d", "LMT_LockheedMartin_ret_1d__macross__MCD_ret_1d", "BLK_BlackRock_zscore_60d__ret5x__AMD_ret_5d", "VIX_Price_ret_5d__ret5x__LMT_LockheedMartin_ret_1d", "heston_var_ev_h7", "VIX_Price_ret_5d__prod__LMT_LockheedMartin_ret_1d"], "is_new": false}, {"model_id": "v1_h1_STRESS_GradientBoosting_N12", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 1, "n_features": 12, "F1_dir": 0.5386, "F1_UP_FORT": 0.3353, "F1_DOWN_FORT": 0.4737, "train_start": "2001-02-08", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["ENB_EnbridgeInc_ret_1d__macross__TXN_ret_20d", "ETN_Eaton_ret_1d__div__US6M_Rate_ret_1d", "egarch_spx_delta_h1__macross__VRTX_VertexPharm_ret_1d", "egarch_spx_delta_h1__div__LMT_LockheedMartin_ret_1d", "EXC_Exelon_zscore_60d", "vix_mean_abs_ret_5d", "LMT_LockheedMartin_ret_1d__macross__MCD_ret_1d", "BLK_BlackRock_zscore_60d__ret5x__AMD_ret_5d", "VIX_Price_ret_5d__ret5x__LMT_LockheedMartin_ret_1d", "heston_var_ev_h7", "VIX_Price_ret_5d__prod__LMT_LockheedMartin_ret_1d", "ETN_Eaton_ret_1d__macross__LMT_LockheedMartin_ret_1d"], "is_new": false}, {"model_id": "v1_h1_STRESS_GradientBoosting_N13", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 1, "n_features": 13, "F1_dir": 0.5555, "F1_UP_FORT": 0.3275, "F1_DOWN_FORT": 0.4273, "train_start": "2001-02-08", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["ENB_EnbridgeInc_ret_1d__macross__TXN_ret_20d", "ETN_Eaton_ret_1d__div__US6M_Rate_ret_1d", "egarch_spx_delta_h1__macross__VRTX_VertexPharm_ret_1d", "egarch_spx_delta_h1__div__LMT_LockheedMartin_ret_1d", "EXC_Exelon_zscore_60d", "vix_mean_abs_ret_5d", "LMT_LockheedMartin_ret_1d__macross__MCD_ret_1d", "BLK_BlackRock_zscore_60d__ret5x__AMD_ret_5d", "VIX_Price_ret_5d__ret5x__LMT_LockheedMartin_ret_1d", "heston_var_ev_h7", "VIX_Price_ret_5d__prod__LMT_LockheedMartin_ret_1d", "ETN_Eaton_ret_1d__macross__LMT_LockheedMartin_ret_1d", "XEL_Xcel_ret_1d__ret5x__DE_Deere_ret_1d"], "is_new": false}, {"model_id": "v1_h1_STRESS_GradientBoosting_N14", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 1, "n_features": 14, "F1_dir": 0.5485, "F1_UP_FORT": 0.3468, "F1_DOWN_FORT": 0.4533, "train_start": "2001-02-08", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["ENB_EnbridgeInc_ret_1d__macross__TXN_ret_20d", "ETN_Eaton_ret_1d__div__US6M_Rate_ret_1d", "egarch_spx_delta_h1__macross__VRTX_VertexPharm_ret_1d", "egarch_spx_delta_h1__div__LMT_LockheedMartin_ret_1d", "EXC_Exelon_zscore_60d", "vix_mean_abs_ret_5d", "LMT_LockheedMartin_ret_1d__macross__MCD_ret_1d", "BLK_BlackRock_zscore_60d__ret5x__AMD_ret_5d", "VIX_Price_ret_5d__ret5x__LMT_LockheedMartin_ret_1d", "heston_var_ev_h7", "VIX_Price_ret_5d__prod__LMT_LockheedMartin_ret_1d", "ETN_Eaton_ret_1d__macross__LMT_LockheedMartin_ret_1d", "XEL_Xcel_ret_1d__ret5x__DE_Deere_ret_1d", "CMCSA_ret_1d"], "is_new": false}, {"model_id": "v1_h1_STRESS_GradientBoosting_N15", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 1, "n_features": 15, "F1_dir": 0.5362, "F1_UP_FORT": 0.3596, "F1_DOWN_FORT": 0.4464, "train_start": "2001-02-08", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["ENB_EnbridgeInc_ret_1d__macross__TXN_ret_20d", "ETN_Eaton_ret_1d__div__US6M_Rate_ret_1d", "egarch_spx_delta_h1__macross__VRTX_VertexPharm_ret_1d", "egarch_spx_delta_h1__div__LMT_LockheedMartin_ret_1d", "EXC_Exelon_zscore_60d", "vix_mean_abs_ret_5d", "LMT_LockheedMartin_ret_1d__macross__MCD_ret_1d", "BLK_BlackRock_zscore_60d__ret5x__AMD_ret_5d", "VIX_Price_ret_5d__ret5x__LMT_LockheedMartin_ret_1d", "heston_var_ev_h7", "VIX_Price_ret_5d__prod__LMT_LockheedMartin_ret_1d", "ETN_Eaton_ret_1d__macross__LMT_LockheedMartin_ret_1d", "XEL_Xcel_ret_1d__ret5x__DE_Deere_ret_1d", "CMCSA_ret_1d", "heston_var_ev_h3"], "is_new": false}, {"model_id": "v1_h1_STRESS_GradientBoosting_N16", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 1, "n_features": 16, "F1_dir": 0.5756, "F1_UP_FORT": 0.3976, "F1_DOWN_FORT": 0.4783, "train_start": "2001-02-08", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["ENB_EnbridgeInc_ret_1d__macross__TXN_ret_20d", "ETN_Eaton_ret_1d__div__US6M_Rate_ret_1d", "egarch_spx_delta_h1__macross__VRTX_VertexPharm_ret_1d", "egarch_spx_delta_h1__div__LMT_LockheedMartin_ret_1d", "EXC_Exelon_zscore_60d", "vix_mean_abs_ret_5d", "LMT_LockheedMartin_ret_1d__macross__MCD_ret_1d", "BLK_BlackRock_zscore_60d__ret5x__AMD_ret_5d", "VIX_Price_ret_5d__ret5x__LMT_LockheedMartin_ret_1d", "heston_var_ev_h7", "VIX_Price_ret_5d__prod__LMT_LockheedMartin_ret_1d", "ETN_Eaton_ret_1d__macross__LMT_LockheedMartin_ret_1d", "XEL_Xcel_ret_1d__ret5x__DE_Deere_ret_1d", "CMCSA_ret_1d", "heston_var_ev_h3", "CPB_CampbellSoup_ret_5d"], "is_new": false}, {"model_id": "v1_h1_STRESS_GradientBoosting_N17", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 1, "n_features": 17, "F1_dir": 0.5319, "F1_UP_FORT": 0.2642, "F1_DOWN_FORT": 0.4816, "train_start": "2001-02-08", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["ENB_EnbridgeInc_ret_1d__macross__TXN_ret_20d", "ETN_Eaton_ret_1d__div__US6M_Rate_ret_1d", "egarch_spx_delta_h1__macross__VRTX_VertexPharm_ret_1d", "egarch_spx_delta_h1__div__LMT_LockheedMartin_ret_1d", "EXC_Exelon_zscore_60d", "vix_mean_abs_ret_5d", "LMT_LockheedMartin_ret_1d__macross__MCD_ret_1d", "BLK_BlackRock_zscore_60d__ret5x__AMD_ret_5d", "VIX_Price_ret_5d__ret5x__LMT_LockheedMartin_ret_1d", "heston_var_ev_h7", "VIX_Price_ret_5d__prod__LMT_LockheedMartin_ret_1d", "ETN_Eaton_ret_1d__macross__LMT_LockheedMartin_ret_1d", "XEL_Xcel_ret_1d__ret5x__DE_Deere_ret_1d", "CMCSA_ret_1d", "heston_var_ev_h3", "CPB_CampbellSoup_ret_5d", "EWJ_Japan_vol_20d"], "is_new": false}, {"model_id": "v1_h1_STRESS_GradientBoosting_N18", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 1, "n_features": 18, "F1_dir": 0.5443, "F1_UP_FORT": 0.3012, "F1_DOWN_FORT": 0.4664, "train_start": "2001-02-08", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["ENB_EnbridgeInc_ret_1d__macross__TXN_ret_20d", "ETN_Eaton_ret_1d__div__US6M_Rate_ret_1d", "egarch_spx_delta_h1__macross__VRTX_VertexPharm_ret_1d", "egarch_spx_delta_h1__div__LMT_LockheedMartin_ret_1d", "EXC_Exelon_zscore_60d", "vix_mean_abs_ret_5d", "LMT_LockheedMartin_ret_1d__macross__MCD_ret_1d", "BLK_BlackRock_zscore_60d__ret5x__AMD_ret_5d", "VIX_Price_ret_5d__ret5x__LMT_LockheedMartin_ret_1d", "heston_var_ev_h7", "VIX_Price_ret_5d__prod__LMT_LockheedMartin_ret_1d", "ETN_Eaton_ret_1d__macross__LMT_LockheedMartin_ret_1d", "XEL_Xcel_ret_1d__ret5x__DE_Deere_ret_1d", "CMCSA_ret_1d", "heston_var_ev_h3", "CPB_CampbellSoup_ret_5d", "EWJ_Japan_vol_20d", "ENB_EnbridgeInc_ret_1d__ret5x__XEL_Xcel_ret_1d"], "is_new": false}, {"model_id": "v1_h1_STRESS_GradientBoosting_N19", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 1, "n_features": 19, "F1_dir": 0.5596, "F1_UP_FORT": 0.3, "F1_DOWN_FORT": 0.5322, "train_start": "2001-02-08", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["ENB_EnbridgeInc_ret_1d__macross__TXN_ret_20d", "ETN_Eaton_ret_1d__div__US6M_Rate_ret_1d", "egarch_spx_delta_h1__macross__VRTX_VertexPharm_ret_1d", "egarch_spx_delta_h1__div__LMT_LockheedMartin_ret_1d", "EXC_Exelon_zscore_60d", "vix_mean_abs_ret_5d", "LMT_LockheedMartin_ret_1d__macross__MCD_ret_1d", "BLK_BlackRock_zscore_60d__ret5x__AMD_ret_5d", "VIX_Price_ret_5d__ret5x__LMT_LockheedMartin_ret_1d", "heston_var_ev_h7", "VIX_Price_ret_5d__prod__LMT_LockheedMartin_ret_1d", "ETN_Eaton_ret_1d__macross__LMT_LockheedMartin_ret_1d", "XEL_Xcel_ret_1d__ret5x__DE_Deere_ret_1d", "CMCSA_ret_1d", "heston_var_ev_h3", "CPB_CampbellSoup_ret_5d", "EWJ_Japan_vol_20d", "ENB_EnbridgeInc_ret_1d__ret5x__XEL_Xcel_ret_1d", "vix_vs_ma20__ret5x__VRTX_VertexPharm_ret_1d"], "is_new": false}, {"model_id": "v1_h1_STRESS_GradientBoosting_N20", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 1, "n_features": 20, "F1_dir": 0.5556, "F1_UP_FORT": 0.325, "F1_DOWN_FORT": 0.4681, "train_start": "2001-02-08", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["ENB_EnbridgeInc_ret_1d__macross__TXN_ret_20d", "ETN_Eaton_ret_1d__div__US6M_Rate_ret_1d", "egarch_spx_delta_h1__macross__VRTX_VertexPharm_ret_1d", "egarch_spx_delta_h1__div__LMT_LockheedMartin_ret_1d", "EXC_Exelon_zscore_60d", "vix_mean_abs_ret_5d", "LMT_LockheedMartin_ret_1d__macross__MCD_ret_1d", "BLK_BlackRock_zscore_60d__ret5x__AMD_ret_5d", "VIX_Price_ret_5d__ret5x__LMT_LockheedMartin_ret_1d", "heston_var_ev_h7", "VIX_Price_ret_5d__prod__LMT_LockheedMartin_ret_1d", "ETN_Eaton_ret_1d__macross__LMT_LockheedMartin_ret_1d", "XEL_Xcel_ret_1d__ret5x__DE_Deere_ret_1d", "CMCSA_ret_1d", "heston_var_ev_h3", "CPB_CampbellSoup_ret_5d", "EWJ_Japan_vol_20d", "ENB_EnbridgeInc_ret_1d__ret5x__XEL_Xcel_ret_1d", "vix_vs_ma20__ret5x__VRTX_VertexPharm_ret_1d", "PPL_PPL_ret_1d__div__BLK_BlackRock_zscore_60d"], "is_new": false}, {"model_id": "v1_h1_STRESS_GradientBoosting_N21", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 1, "n_features": 21, "F1_dir": 0.5433, "F1_UP_FORT": 0.3509, "F1_DOWN_FORT": 0.469, "train_start": "2001-02-08", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["ENB_EnbridgeInc_ret_1d__macross__TXN_ret_20d", "ETN_Eaton_ret_1d__div__US6M_Rate_ret_1d", "egarch_spx_delta_h1__macross__VRTX_VertexPharm_ret_1d", "egarch_spx_delta_h1__div__LMT_LockheedMartin_ret_1d", "EXC_Exelon_zscore_60d", "vix_mean_abs_ret_5d", "LMT_LockheedMartin_ret_1d__macross__MCD_ret_1d", "BLK_BlackRock_zscore_60d__ret5x__AMD_ret_5d", "VIX_Price_ret_5d__ret5x__LMT_LockheedMartin_ret_1d", "heston_var_ev_h7", "VIX_Price_ret_5d__prod__LMT_LockheedMartin_ret_1d", "ETN_Eaton_ret_1d__macross__LMT_LockheedMartin_ret_1d", "XEL_Xcel_ret_1d__ret5x__DE_Deere_ret_1d", "CMCSA_ret_1d", "heston_var_ev_h3", "CPB_CampbellSoup_ret_5d", "EWJ_Japan_vol_20d", "ENB_EnbridgeInc_ret_1d__ret5x__XEL_Xcel_ret_1d", "vix_vs_ma20__ret5x__VRTX_VertexPharm_ret_1d", "PPL_PPL_ret_1d__div__BLK_BlackRock_zscore_60d", "NFCI_zscore_60d__macross__VRTX_VertexPharm_ret_1d"], "is_new": false}, {"model_id": "v1_h1_STRESS_GradientBoosting_N22", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 1, "n_features": 22, "F1_dir": 0.5272, "F1_UP_FORT": 0.2651, "F1_DOWN_FORT": 0.4513, "train_start": "2001-02-08", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["ENB_EnbridgeInc_ret_1d__macross__TXN_ret_20d", "ETN_Eaton_ret_1d__div__US6M_Rate_ret_1d", "egarch_spx_delta_h1__macross__VRTX_VertexPharm_ret_1d", "egarch_spx_delta_h1__div__LMT_LockheedMartin_ret_1d", "EXC_Exelon_zscore_60d", "vix_mean_abs_ret_5d", "LMT_LockheedMartin_ret_1d__macross__MCD_ret_1d", "BLK_BlackRock_zscore_60d__ret5x__AMD_ret_5d", "VIX_Price_ret_5d__ret5x__LMT_LockheedMartin_ret_1d", "heston_var_ev_h7", "VIX_Price_ret_5d__prod__LMT_LockheedMartin_ret_1d", "ETN_Eaton_ret_1d__macross__LMT_LockheedMartin_ret_1d", "XEL_Xcel_ret_1d__ret5x__DE_Deere_ret_1d", "CMCSA_ret_1d", "heston_var_ev_h3", "CPB_CampbellSoup_ret_5d", "EWJ_Japan_vol_20d", "ENB_EnbridgeInc_ret_1d__ret5x__XEL_Xcel_ret_1d", "vix_vs_ma20__ret5x__VRTX_VertexPharm_ret_1d", "PPL_PPL_ret_1d__div__BLK_BlackRock_zscore_60d", "NFCI_zscore_60d__macross__VRTX_VertexPharm_ret_1d", "PCAR_PaccarInc_ret_5d"], "is_new": false}, {"model_id": "v1_h1_STRESS_RandomForest_N6", "algo": "RandomForest", "regime": "STRESS", "horizon": 1, "n_features": 6, "F1_dir": 0.5075, "F1_UP_FORT": 0.2361, "F1_DOWN_FORT": 0.472, "train_start": "2001-02-08", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["ENB_EnbridgeInc_ret_1d__macross__TXN_ret_20d", "ETN_Eaton_ret_1d__div__US6M_Rate_ret_1d", "egarch_spx_delta_h1__macross__VRTX_VertexPharm_ret_1d", "egarch_spx_delta_h1__div__LMT_LockheedMartin_ret_1d", "EXC_Exelon_zscore_60d", "vix_mean_abs_ret_5d"], "is_new": false}, {"model_id": "v1_h1_STRESS_RandomForest_N7", "algo": "RandomForest", "regime": "STRESS", "horizon": 1, "n_features": 7, "F1_dir": 0.5208, "F1_UP_FORT": 0.2256, "F1_DOWN_FORT": 0.48, "train_start": "2001-02-08", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["ENB_EnbridgeInc_ret_1d__macross__TXN_ret_20d", "ETN_Eaton_ret_1d__div__US6M_Rate_ret_1d", "egarch_spx_delta_h1__macross__VRTX_VertexPharm_ret_1d", "egarch_spx_delta_h1__div__LMT_LockheedMartin_ret_1d", "EXC_Exelon_zscore_60d", "vix_mean_abs_ret_5d", "LMT_LockheedMartin_ret_1d__macross__MCD_ret_1d"], "is_new": false}, {"model_id": "v1_h1_STRESS_RandomForest_N8", "algo": "RandomForest", "regime": "STRESS", "horizon": 1, "n_features": 8, "F1_dir": 0.526, "F1_UP_FORT": 0.2137, "F1_DOWN_FORT": 0.4839, "train_start": "2001-02-08", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["ENB_EnbridgeInc_ret_1d__macross__TXN_ret_20d", "ETN_Eaton_ret_1d__div__US6M_Rate_ret_1d", "egarch_spx_delta_h1__macross__VRTX_VertexPharm_ret_1d", "egarch_spx_delta_h1__div__LMT_LockheedMartin_ret_1d", "EXC_Exelon_zscore_60d", "vix_mean_abs_ret_5d", "LMT_LockheedMartin_ret_1d__macross__MCD_ret_1d", "BLK_BlackRock_zscore_60d__ret5x__AMD_ret_5d"], "is_new": false}, {"model_id": "v1_h1_STRESS_RandomForest_N9", "algo": "RandomForest", "regime": "STRESS", "horizon": 1, "n_features": 9, "F1_dir": 0.5448, "F1_UP_FORT": 0.2721, "F1_DOWN_FORT": 0.5041, "train_start": "2001-02-08", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["ENB_EnbridgeInc_ret_1d__macross__TXN_ret_20d", "ETN_Eaton_ret_1d__div__US6M_Rate_ret_1d", "egarch_spx_delta_h1__macross__VRTX_VertexPharm_ret_1d", "egarch_spx_delta_h1__div__LMT_LockheedMartin_ret_1d", "EXC_Exelon_zscore_60d", "vix_mean_abs_ret_5d", "LMT_LockheedMartin_ret_1d__macross__MCD_ret_1d", "BLK_BlackRock_zscore_60d__ret5x__AMD_ret_5d", "VIX_Price_ret_5d__ret5x__LMT_LockheedMartin_ret_1d"], "is_new": false}, {"model_id": "v1_h1_STRESS_RandomForest_N11", "algo": "RandomForest", "regime": "STRESS", "horizon": 1, "n_features": 11, "F1_dir": 0.516, "F1_UP_FORT": 0.2676, "F1_DOWN_FORT": 0.5211, "train_start": "2001-02-08", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["ENB_EnbridgeInc_ret_1d__macross__TXN_ret_20d", "ETN_Eaton_ret_1d__div__US6M_Rate_ret_1d", "egarch_spx_delta_h1__macross__VRTX_VertexPharm_ret_1d", "egarch_spx_delta_h1__div__LMT_LockheedMartin_ret_1d", "EXC_Exelon_zscore_60d", "vix_mean_abs_ret_5d", "LMT_LockheedMartin_ret_1d__macross__MCD_ret_1d", "BLK_BlackRock_zscore_60d__ret5x__AMD_ret_5d", "VIX_Price_ret_5d__ret5x__LMT_LockheedMartin_ret_1d", "heston_var_ev_h7", "VIX_Price_ret_5d__prod__LMT_LockheedMartin_ret_1d"], "is_new": false}, {"model_id": "v1_h1_STRESS_RandomForest_N12", "algo": "RandomForest", "regime": "STRESS", "horizon": 1, "n_features": 12, "F1_dir": 0.5245, "F1_UP_FORT": 0.2388, "F1_DOWN_FORT": 0.5407, "train_start": "2001-02-08", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["ENB_EnbridgeInc_ret_1d__macross__TXN_ret_20d", "ETN_Eaton_ret_1d__div__US6M_Rate_ret_1d", "egarch_spx_delta_h1__macross__VRTX_VertexPharm_ret_1d", "egarch_spx_delta_h1__div__LMT_LockheedMartin_ret_1d", "EXC_Exelon_zscore_60d", "vix_mean_abs_ret_5d", "LMT_LockheedMartin_ret_1d__macross__MCD_ret_1d", "BLK_BlackRock_zscore_60d__ret5x__AMD_ret_5d", "VIX_Price_ret_5d__ret5x__LMT_LockheedMartin_ret_1d", "heston_var_ev_h7", "VIX_Price_ret_5d__prod__LMT_LockheedMartin_ret_1d", "ETN_Eaton_ret_1d__macross__LMT_LockheedMartin_ret_1d"], "is_new": false}, {"model_id": "v1_h1_STRESS_RandomForest_N13", "algo": "RandomForest", "regime": "STRESS", "horizon": 1, "n_features": 13, "F1_dir": 0.5227, "F1_UP_FORT": 0.2344, "F1_DOWN_FORT": 0.5275, "train_start": "2001-02-08", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["ENB_EnbridgeInc_ret_1d__macross__TXN_ret_20d", "ETN_Eaton_ret_1d__div__US6M_Rate_ret_1d", "egarch_spx_delta_h1__macross__VRTX_VertexPharm_ret_1d", "egarch_spx_delta_h1__div__LMT_LockheedMartin_ret_1d", "EXC_Exelon_zscore_60d", "vix_mean_abs_ret_5d", "LMT_LockheedMartin_ret_1d__macross__MCD_ret_1d", "BLK_BlackRock_zscore_60d__ret5x__AMD_ret_5d", "VIX_Price_ret_5d__ret5x__LMT_LockheedMartin_ret_1d", "heston_var_ev_h7", "VIX_Price_ret_5d__prod__LMT_LockheedMartin_ret_1d", "ETN_Eaton_ret_1d__macross__LMT_LockheedMartin_ret_1d", "XEL_Xcel_ret_1d__ret5x__DE_Deere_ret_1d"], "is_new": false}, {"model_id": "v1_h1_STRESS_RandomForest_N14", "algo": "RandomForest", "regime": "STRESS", "horizon": 1, "n_features": 14, "F1_dir": 0.5433, "F1_UP_FORT": 0.28, "F1_DOWN_FORT": 0.5098, "train_start": "2001-02-08", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["ENB_EnbridgeInc_ret_1d__macross__TXN_ret_20d", "ETN_Eaton_ret_1d__div__US6M_Rate_ret_1d", "egarch_spx_delta_h1__macross__VRTX_VertexPharm_ret_1d", "egarch_spx_delta_h1__div__LMT_LockheedMartin_ret_1d", "EXC_Exelon_zscore_60d", "vix_mean_abs_ret_5d", "LMT_LockheedMartin_ret_1d__macross__MCD_ret_1d", "BLK_BlackRock_zscore_60d__ret5x__AMD_ret_5d", "VIX_Price_ret_5d__ret5x__LMT_LockheedMartin_ret_1d", "heston_var_ev_h7", "VIX_Price_ret_5d__prod__LMT_LockheedMartin_ret_1d", "ETN_Eaton_ret_1d__macross__LMT_LockheedMartin_ret_1d", "XEL_Xcel_ret_1d__ret5x__DE_Deere_ret_1d", "CMCSA_ret_1d"], "is_new": false}, {"model_id": "v1_h1_STRESS_RandomForest_N15", "algo": "RandomForest", "regime": "STRESS", "horizon": 1, "n_features": 15, "F1_dir": 0.5291, "F1_UP_FORT": 0.1732, "F1_DOWN_FORT": 0.5319, "train_start": "2001-02-08", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["ENB_EnbridgeInc_ret_1d__macross__TXN_ret_20d", "ETN_Eaton_ret_1d__div__US6M_Rate_ret_1d", "egarch_spx_delta_h1__macross__VRTX_VertexPharm_ret_1d", "egarch_spx_delta_h1__div__LMT_LockheedMartin_ret_1d", "EXC_Exelon_zscore_60d", "vix_mean_abs_ret_5d", "LMT_LockheedMartin_ret_1d__macross__MCD_ret_1d", "BLK_BlackRock_zscore_60d__ret5x__AMD_ret_5d", "VIX_Price_ret_5d__ret5x__LMT_LockheedMartin_ret_1d", "heston_var_ev_h7", "VIX_Price_ret_5d__prod__LMT_LockheedMartin_ret_1d", "ETN_Eaton_ret_1d__macross__LMT_LockheedMartin_ret_1d", "XEL_Xcel_ret_1d__ret5x__DE_Deere_ret_1d", "CMCSA_ret_1d", "heston_var_ev_h3"], "is_new": false}, {"model_id": "v1_h1_STRESS_RandomForest_N16", "algo": "RandomForest", "regime": "STRESS", "horizon": 1, "n_features": 16, "F1_dir": 0.5158, "F1_UP_FORT": 0.2406, "F1_DOWN_FORT": 0.5128, "train_start": "2001-02-08", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["ENB_EnbridgeInc_ret_1d__macross__TXN_ret_20d", "ETN_Eaton_ret_1d__div__US6M_Rate_ret_1d", "egarch_spx_delta_h1__macross__VRTX_VertexPharm_ret_1d", "egarch_spx_delta_h1__div__LMT_LockheedMartin_ret_1d", "EXC_Exelon_zscore_60d", "vix_mean_abs_ret_5d", "LMT_LockheedMartin_ret_1d__macross__MCD_ret_1d", "BLK_BlackRock_zscore_60d__ret5x__AMD_ret_5d", "VIX_Price_ret_5d__ret5x__LMT_LockheedMartin_ret_1d", "heston_var_ev_h7", "VIX_Price_ret_5d__prod__LMT_LockheedMartin_ret_1d", "ETN_Eaton_ret_1d__macross__LMT_LockheedMartin_ret_1d", "XEL_Xcel_ret_1d__ret5x__DE_Deere_ret_1d", "CMCSA_ret_1d", "heston_var_ev_h3", "CPB_CampbellSoup_ret_5d"], "is_new": false}, {"model_id": "v1_h1_STRESS_RandomForest_N17", "algo": "RandomForest", "regime": "STRESS", "horizon": 1, "n_features": 17, "F1_dir": 0.5218, "F1_UP_FORT": 0.1833, "F1_DOWN_FORT": 0.5357, "train_start": "2001-02-08", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["ENB_EnbridgeInc_ret_1d__macross__TXN_ret_20d", "ETN_Eaton_ret_1d__div__US6M_Rate_ret_1d", "egarch_spx_delta_h1__macross__VRTX_VertexPharm_ret_1d", "egarch_spx_delta_h1__div__LMT_LockheedMartin_ret_1d", "EXC_Exelon_zscore_60d", "vix_mean_abs_ret_5d", "LMT_LockheedMartin_ret_1d__macross__MCD_ret_1d", "BLK_BlackRock_zscore_60d__ret5x__AMD_ret_5d", "VIX_Price_ret_5d__ret5x__LMT_LockheedMartin_ret_1d", "heston_var_ev_h7", "VIX_Price_ret_5d__prod__LMT_LockheedMartin_ret_1d", "ETN_Eaton_ret_1d__macross__LMT_LockheedMartin_ret_1d", "XEL_Xcel_ret_1d__ret5x__DE_Deere_ret_1d", "CMCSA_ret_1d", "heston_var_ev_h3", "CPB_CampbellSoup_ret_5d", "EWJ_Japan_vol_20d"], "is_new": false}, {"model_id": "v1_h1_STRESS_RandomForest_N18", "algo": "RandomForest", "regime": "STRESS", "horizon": 1, "n_features": 18, "F1_dir": 0.5308, "F1_UP_FORT": 0.2167, "F1_DOWN_FORT": 0.5248, "train_start": "2001-02-08", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["ENB_EnbridgeInc_ret_1d__macross__TXN_ret_20d", "ETN_Eaton_ret_1d__div__US6M_Rate_ret_1d", "egarch_spx_delta_h1__macross__VRTX_VertexPharm_ret_1d", "egarch_spx_delta_h1__div__LMT_LockheedMartin_ret_1d", "EXC_Exelon_zscore_60d", "vix_mean_abs_ret_5d", "LMT_LockheedMartin_ret_1d__macross__MCD_ret_1d", "BLK_BlackRock_zscore_60d__ret5x__AMD_ret_5d", "VIX_Price_ret_5d__ret5x__LMT_LockheedMartin_ret_1d", "heston_var_ev_h7", "VIX_Price_ret_5d__prod__LMT_LockheedMartin_ret_1d", "ETN_Eaton_ret_1d__macross__LMT_LockheedMartin_ret_1d", "XEL_Xcel_ret_1d__ret5x__DE_Deere_ret_1d", "CMCSA_ret_1d", "heston_var_ev_h3", "CPB_CampbellSoup_ret_5d", "EWJ_Japan_vol_20d", "ENB_EnbridgeInc_ret_1d__ret5x__XEL_Xcel_ret_1d"], "is_new": false}, {"model_id": "v1_h1_STRESS_RandomForest_N19", "algo": "RandomForest", "regime": "STRESS", "horizon": 1, "n_features": 19, "F1_dir": 0.5193, "F1_UP_FORT": 0.192, "F1_DOWN_FORT": 0.5357, "train_start": "2001-02-08", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["ENB_EnbridgeInc_ret_1d__macross__TXN_ret_20d", "ETN_Eaton_ret_1d__div__US6M_Rate_ret_1d", "egarch_spx_delta_h1__macross__VRTX_VertexPharm_ret_1d", "egarch_spx_delta_h1__div__LMT_LockheedMartin_ret_1d", "EXC_Exelon_zscore_60d", "vix_mean_abs_ret_5d", "LMT_LockheedMartin_ret_1d__macross__MCD_ret_1d", "BLK_BlackRock_zscore_60d__ret5x__AMD_ret_5d", "VIX_Price_ret_5d__ret5x__LMT_LockheedMartin_ret_1d", "heston_var_ev_h7", "VIX_Price_ret_5d__prod__LMT_LockheedMartin_ret_1d", "ETN_Eaton_ret_1d__macross__LMT_LockheedMartin_ret_1d", "XEL_Xcel_ret_1d__ret5x__DE_Deere_ret_1d", "CMCSA_ret_1d", "heston_var_ev_h3", "CPB_CampbellSoup_ret_5d", "EWJ_Japan_vol_20d", "ENB_EnbridgeInc_ret_1d__ret5x__XEL_Xcel_ret_1d", "vix_vs_ma20__ret5x__VRTX_VertexPharm_ret_1d"], "is_new": false}, {"model_id": "v1_h1_STRESS_RandomForest_N20", "algo": "RandomForest", "regime": "STRESS", "horizon": 1, "n_features": 20, "F1_dir": 0.5, "F1_UP_FORT": 0.1935, "F1_DOWN_FORT": 0.5199, "train_start": "2001-02-08", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["ENB_EnbridgeInc_ret_1d__macross__TXN_ret_20d", "ETN_Eaton_ret_1d__div__US6M_Rate_ret_1d", "egarch_spx_delta_h1__macross__VRTX_VertexPharm_ret_1d", "egarch_spx_delta_h1__div__LMT_LockheedMartin_ret_1d", "EXC_Exelon_zscore_60d", "vix_mean_abs_ret_5d", "LMT_LockheedMartin_ret_1d__macross__MCD_ret_1d", "BLK_BlackRock_zscore_60d__ret5x__AMD_ret_5d", "VIX_Price_ret_5d__ret5x__LMT_LockheedMartin_ret_1d", "heston_var_ev_h7", "VIX_Price_ret_5d__prod__LMT_LockheedMartin_ret_1d", "ETN_Eaton_ret_1d__macross__LMT_LockheedMartin_ret_1d", "XEL_Xcel_ret_1d__ret5x__DE_Deere_ret_1d", "CMCSA_ret_1d", "heston_var_ev_h3", "CPB_CampbellSoup_ret_5d", "EWJ_Japan_vol_20d", "ENB_EnbridgeInc_ret_1d__ret5x__XEL_Xcel_ret_1d", "vix_vs_ma20__ret5x__VRTX_VertexPharm_ret_1d", "PPL_PPL_ret_1d__div__BLK_BlackRock_zscore_60d"], "is_new": false}, {"model_id": "v1_h1_STRESS_RandomForest_N21", "algo": "RandomForest", "regime": "STRESS", "horizon": 1, "n_features": 21, "F1_dir": 0.5349, "F1_UP_FORT": 0.2314, "F1_DOWN_FORT": 0.5409, "train_start": "2001-02-08", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["ENB_EnbridgeInc_ret_1d__macross__TXN_ret_20d", "ETN_Eaton_ret_1d__div__US6M_Rate_ret_1d", "egarch_spx_delta_h1__macross__VRTX_VertexPharm_ret_1d", "egarch_spx_delta_h1__div__LMT_LockheedMartin_ret_1d", "EXC_Exelon_zscore_60d", "vix_mean_abs_ret_5d", "LMT_LockheedMartin_ret_1d__macross__MCD_ret_1d", "BLK_BlackRock_zscore_60d__ret5x__AMD_ret_5d", "VIX_Price_ret_5d__ret5x__LMT_LockheedMartin_ret_1d", "heston_var_ev_h7", "VIX_Price_ret_5d__prod__LMT_LockheedMartin_ret_1d", "ETN_Eaton_ret_1d__macross__LMT_LockheedMartin_ret_1d", "XEL_Xcel_ret_1d__ret5x__DE_Deere_ret_1d", "CMCSA_ret_1d", "heston_var_ev_h3", "CPB_CampbellSoup_ret_5d", "EWJ_Japan_vol_20d", "ENB_EnbridgeInc_ret_1d__ret5x__XEL_Xcel_ret_1d", "vix_vs_ma20__ret5x__VRTX_VertexPharm_ret_1d", "PPL_PPL_ret_1d__div__BLK_BlackRock_zscore_60d", "NFCI_zscore_60d__macross__VRTX_VertexPharm_ret_1d"], "is_new": false}, {"model_id": "v1_h1_STRESS_RandomForest_N22", "algo": "RandomForest", "regime": "STRESS", "horizon": 1, "n_features": 22, "F1_dir": 0.5218, "F1_UP_FORT": 0.2656, "F1_DOWN_FORT": 0.5435, "train_start": "2001-02-08", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["ENB_EnbridgeInc_ret_1d__macross__TXN_ret_20d", "ETN_Eaton_ret_1d__div__US6M_Rate_ret_1d", "egarch_spx_delta_h1__macross__VRTX_VertexPharm_ret_1d", "egarch_spx_delta_h1__div__LMT_LockheedMartin_ret_1d", "EXC_Exelon_zscore_60d", "vix_mean_abs_ret_5d", "LMT_LockheedMartin_ret_1d__macross__MCD_ret_1d", "BLK_BlackRock_zscore_60d__ret5x__AMD_ret_5d", "VIX_Price_ret_5d__ret5x__LMT_LockheedMartin_ret_1d", "heston_var_ev_h7", "VIX_Price_ret_5d__prod__LMT_LockheedMartin_ret_1d", "ETN_Eaton_ret_1d__macross__LMT_LockheedMartin_ret_1d", "XEL_Xcel_ret_1d__ret5x__DE_Deere_ret_1d", "CMCSA_ret_1d", "heston_var_ev_h3", "CPB_CampbellSoup_ret_5d", "EWJ_Japan_vol_20d", "ENB_EnbridgeInc_ret_1d__ret5x__XEL_Xcel_ret_1d", "vix_vs_ma20__ret5x__VRTX_VertexPharm_ret_1d", "PPL_PPL_ret_1d__div__BLK_BlackRock_zscore_60d", "NFCI_zscore_60d__macross__VRTX_VertexPharm_ret_1d", "PCAR_PaccarInc_ret_5d"], "is_new": false}, {"model_id": "v1_h1_STRESS_LogisticRegression_N8", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 1, "n_features": 8, "F1_dir": 0.5127, "F1_UP_FORT": 0.2667, "F1_DOWN_FORT": 0.4711, "train_start": "2001-02-08", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["ENB_EnbridgeInc_ret_1d__macross__TXN_ret_20d", "ETN_Eaton_ret_1d__div__US6M_Rate_ret_1d", "egarch_spx_delta_h1__macross__VRTX_VertexPharm_ret_1d", "egarch_spx_delta_h1__div__LMT_LockheedMartin_ret_1d", "EXC_Exelon_zscore_60d", "vix_mean_abs_ret_5d", "LMT_LockheedMartin_ret_1d__macross__MCD_ret_1d", "BLK_BlackRock_zscore_60d__ret5x__AMD_ret_5d"], "is_new": false}, {"model_id": "v1_h1_STRESS_LogisticRegression_N9", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 1, "n_features": 9, "F1_dir": 0.5223, "F1_UP_FORT": 0.2781, "F1_DOWN_FORT": 0.4865, "train_start": "2001-02-08", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["ENB_EnbridgeInc_ret_1d__macross__TXN_ret_20d", "ETN_Eaton_ret_1d__div__US6M_Rate_ret_1d", "egarch_spx_delta_h1__macross__VRTX_VertexPharm_ret_1d", "egarch_spx_delta_h1__div__LMT_LockheedMartin_ret_1d", "EXC_Exelon_zscore_60d", "vix_mean_abs_ret_5d", "LMT_LockheedMartin_ret_1d__macross__MCD_ret_1d", "BLK_BlackRock_zscore_60d__ret5x__AMD_ret_5d", "VIX_Price_ret_5d__ret5x__LMT_LockheedMartin_ret_1d"], "is_new": false}, {"model_id": "v1_h1_STRESS_LogisticRegression_N10", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 1, "n_features": 10, "F1_dir": 0.5326, "F1_UP_FORT": 0.2857, "F1_DOWN_FORT": 0.5227, "train_start": "2001-02-08", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["ENB_EnbridgeInc_ret_1d__macross__TXN_ret_20d", "ETN_Eaton_ret_1d__div__US6M_Rate_ret_1d", "egarch_spx_delta_h1__macross__VRTX_VertexPharm_ret_1d", "egarch_spx_delta_h1__div__LMT_LockheedMartin_ret_1d", "EXC_Exelon_zscore_60d", "vix_mean_abs_ret_5d", "LMT_LockheedMartin_ret_1d__macross__MCD_ret_1d", "BLK_BlackRock_zscore_60d__ret5x__AMD_ret_5d", "VIX_Price_ret_5d__ret5x__LMT_LockheedMartin_ret_1d", "heston_var_ev_h7"], "is_new": false}, {"model_id": "v1_h1_STRESS_LogisticRegression_N11", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 1, "n_features": 11, "F1_dir": 0.5262, "F1_UP_FORT": 0.2857, "F1_DOWN_FORT": 0.5208, "train_start": "2001-02-08", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["ENB_EnbridgeInc_ret_1d__macross__TXN_ret_20d", "ETN_Eaton_ret_1d__div__US6M_Rate_ret_1d", "egarch_spx_delta_h1__macross__VRTX_VertexPharm_ret_1d", "egarch_spx_delta_h1__div__LMT_LockheedMartin_ret_1d", "EXC_Exelon_zscore_60d", "vix_mean_abs_ret_5d", "LMT_LockheedMartin_ret_1d__macross__MCD_ret_1d", "BLK_BlackRock_zscore_60d__ret5x__AMD_ret_5d", "VIX_Price_ret_5d__ret5x__LMT_LockheedMartin_ret_1d", "heston_var_ev_h7", "VIX_Price_ret_5d__prod__LMT_LockheedMartin_ret_1d"], "is_new": false}, {"model_id": "v1_h1_STRESS_LogisticRegression_N12", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 1, "n_features": 12, "F1_dir": 0.5169, "F1_UP_FORT": 0.2899, "F1_DOWN_FORT": 0.5211, "train_start": "2001-02-08", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["ENB_EnbridgeInc_ret_1d__macross__TXN_ret_20d", "ETN_Eaton_ret_1d__div__US6M_Rate_ret_1d", "egarch_spx_delta_h1__macross__VRTX_VertexPharm_ret_1d", "egarch_spx_delta_h1__div__LMT_LockheedMartin_ret_1d", "EXC_Exelon_zscore_60d", "vix_mean_abs_ret_5d", "LMT_LockheedMartin_ret_1d__macross__MCD_ret_1d", "BLK_BlackRock_zscore_60d__ret5x__AMD_ret_5d", "VIX_Price_ret_5d__ret5x__LMT_LockheedMartin_ret_1d", "heston_var_ev_h7", "VIX_Price_ret_5d__prod__LMT_LockheedMartin_ret_1d", "ETN_Eaton_ret_1d__macross__LMT_LockheedMartin_ret_1d"], "is_new": false}, {"model_id": "v1_h1_STRESS_LogisticRegression_N13", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 1, "n_features": 13, "F1_dir": 0.5112, "F1_UP_FORT": 0.2815, "F1_DOWN_FORT": 0.5208, "train_start": "2001-02-08", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["ENB_EnbridgeInc_ret_1d__macross__TXN_ret_20d", "ETN_Eaton_ret_1d__div__US6M_Rate_ret_1d", "egarch_spx_delta_h1__macross__VRTX_VertexPharm_ret_1d", "egarch_spx_delta_h1__div__LMT_LockheedMartin_ret_1d", "EXC_Exelon_zscore_60d", "vix_mean_abs_ret_5d", "LMT_LockheedMartin_ret_1d__macross__MCD_ret_1d", "BLK_BlackRock_zscore_60d__ret5x__AMD_ret_5d", "VIX_Price_ret_5d__ret5x__LMT_LockheedMartin_ret_1d", "heston_var_ev_h7", "VIX_Price_ret_5d__prod__LMT_LockheedMartin_ret_1d", "ETN_Eaton_ret_1d__macross__LMT_LockheedMartin_ret_1d", "XEL_Xcel_ret_1d__ret5x__DE_Deere_ret_1d"], "is_new": false}, {"model_id": "v1_h1_STRESS_LogisticRegression_N14", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 1, "n_features": 14, "F1_dir": 0.5326, "F1_UP_FORT": 0.2774, "F1_DOWN_FORT": 0.5169, "train_start": "2001-02-08", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["ENB_EnbridgeInc_ret_1d__macross__TXN_ret_20d", "ETN_Eaton_ret_1d__div__US6M_Rate_ret_1d", "egarch_spx_delta_h1__macross__VRTX_VertexPharm_ret_1d", "egarch_spx_delta_h1__div__LMT_LockheedMartin_ret_1d", "EXC_Exelon_zscore_60d", "vix_mean_abs_ret_5d", "LMT_LockheedMartin_ret_1d__macross__MCD_ret_1d", "BLK_BlackRock_zscore_60d__ret5x__AMD_ret_5d", "VIX_Price_ret_5d__ret5x__LMT_LockheedMartin_ret_1d", "heston_var_ev_h7", "VIX_Price_ret_5d__prod__LMT_LockheedMartin_ret_1d", "ETN_Eaton_ret_1d__macross__LMT_LockheedMartin_ret_1d", "XEL_Xcel_ret_1d__ret5x__DE_Deere_ret_1d", "CMCSA_ret_1d"], "is_new": false}, {"model_id": "v1_h1_STRESS_LogisticRegression_N15", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 1, "n_features": 15, "F1_dir": 0.5192, "F1_UP_FORT": 0.2429, "F1_DOWN_FORT": 0.5076, "train_start": "2001-02-08", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["ENB_EnbridgeInc_ret_1d__macross__TXN_ret_20d", "ETN_Eaton_ret_1d__div__US6M_Rate_ret_1d", "egarch_spx_delta_h1__macross__VRTX_VertexPharm_ret_1d", "egarch_spx_delta_h1__div__LMT_LockheedMartin_ret_1d", "EXC_Exelon_zscore_60d", "vix_mean_abs_ret_5d", "LMT_LockheedMartin_ret_1d__macross__MCD_ret_1d", "BLK_BlackRock_zscore_60d__ret5x__AMD_ret_5d", "VIX_Price_ret_5d__ret5x__LMT_LockheedMartin_ret_1d", "heston_var_ev_h7", "VIX_Price_ret_5d__prod__LMT_LockheedMartin_ret_1d", "ETN_Eaton_ret_1d__macross__LMT_LockheedMartin_ret_1d", "XEL_Xcel_ret_1d__ret5x__DE_Deere_ret_1d", "CMCSA_ret_1d", "heston_var_ev_h3"], "is_new": false}, {"model_id": "v1_h1_STRESS_LogisticRegression_N16", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 1, "n_features": 16, "F1_dir": 0.5146, "F1_UP_FORT": 0.2319, "F1_DOWN_FORT": 0.5038, "train_start": "2001-02-08", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["ENB_EnbridgeInc_ret_1d__macross__TXN_ret_20d", "ETN_Eaton_ret_1d__div__US6M_Rate_ret_1d", "egarch_spx_delta_h1__macross__VRTX_VertexPharm_ret_1d", "egarch_spx_delta_h1__div__LMT_LockheedMartin_ret_1d", "EXC_Exelon_zscore_60d", "vix_mean_abs_ret_5d", "LMT_LockheedMartin_ret_1d__macross__MCD_ret_1d", "BLK_BlackRock_zscore_60d__ret5x__AMD_ret_5d", "VIX_Price_ret_5d__ret5x__LMT_LockheedMartin_ret_1d", "heston_var_ev_h7", "VIX_Price_ret_5d__prod__LMT_LockheedMartin_ret_1d", "ETN_Eaton_ret_1d__macross__LMT_LockheedMartin_ret_1d", "XEL_Xcel_ret_1d__ret5x__DE_Deere_ret_1d", "CMCSA_ret_1d", "heston_var_ev_h3", "CPB_CampbellSoup_ret_5d"], "is_new": false}, {"model_id": "v1_h1_STRESS_LogisticRegression_N17", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 1, "n_features": 17, "F1_dir": 0.5239, "F1_UP_FORT": 0.192, "F1_DOWN_FORT": 0.524, "train_start": "2001-02-08", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["ENB_EnbridgeInc_ret_1d__macross__TXN_ret_20d", "ETN_Eaton_ret_1d__div__US6M_Rate_ret_1d", "egarch_spx_delta_h1__macross__VRTX_VertexPharm_ret_1d", "egarch_spx_delta_h1__div__LMT_LockheedMartin_ret_1d", "EXC_Exelon_zscore_60d", "vix_mean_abs_ret_5d", "LMT_LockheedMartin_ret_1d__macross__MCD_ret_1d", "BLK_BlackRock_zscore_60d__ret5x__AMD_ret_5d", "VIX_Price_ret_5d__ret5x__LMT_LockheedMartin_ret_1d", "heston_var_ev_h7", "VIX_Price_ret_5d__prod__LMT_LockheedMartin_ret_1d", "ETN_Eaton_ret_1d__macross__LMT_LockheedMartin_ret_1d", "XEL_Xcel_ret_1d__ret5x__DE_Deere_ret_1d", "CMCSA_ret_1d", "heston_var_ev_h3", "CPB_CampbellSoup_ret_5d", "EWJ_Japan_vol_20d"], "is_new": false}, {"model_id": "v1_h1_STRESS_LogisticRegression_N18", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 1, "n_features": 18, "F1_dir": 0.5342, "F1_UP_FORT": 0.2114, "F1_DOWN_FORT": 0.5314, "train_start": "2001-02-08", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["ENB_EnbridgeInc_ret_1d__macross__TXN_ret_20d", "ETN_Eaton_ret_1d__div__US6M_Rate_ret_1d", "egarch_spx_delta_h1__macross__VRTX_VertexPharm_ret_1d", "egarch_spx_delta_h1__div__LMT_LockheedMartin_ret_1d", "EXC_Exelon_zscore_60d", "vix_mean_abs_ret_5d", "LMT_LockheedMartin_ret_1d__macross__MCD_ret_1d", "BLK_BlackRock_zscore_60d__ret5x__AMD_ret_5d", "VIX_Price_ret_5d__ret5x__LMT_LockheedMartin_ret_1d", "heston_var_ev_h7", "VIX_Price_ret_5d__prod__LMT_LockheedMartin_ret_1d", "ETN_Eaton_ret_1d__macross__LMT_LockheedMartin_ret_1d", "XEL_Xcel_ret_1d__ret5x__DE_Deere_ret_1d", "CMCSA_ret_1d", "heston_var_ev_h3", "CPB_CampbellSoup_ret_5d", "EWJ_Japan_vol_20d", "ENB_EnbridgeInc_ret_1d__ret5x__XEL_Xcel_ret_1d"], "is_new": false}, {"model_id": "v1_h1_STRESS_LogisticRegression_N19", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 1, "n_features": 19, "F1_dir": 0.5342, "F1_UP_FORT": 0.2419, "F1_DOWN_FORT": 0.5314, "train_start": "2001-02-08", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["ENB_EnbridgeInc_ret_1d__macross__TXN_ret_20d", "ETN_Eaton_ret_1d__div__US6M_Rate_ret_1d", "egarch_spx_delta_h1__macross__VRTX_VertexPharm_ret_1d", "egarch_spx_delta_h1__div__LMT_LockheedMartin_ret_1d", "EXC_Exelon_zscore_60d", "vix_mean_abs_ret_5d", "LMT_LockheedMartin_ret_1d__macross__MCD_ret_1d", "BLK_BlackRock_zscore_60d__ret5x__AMD_ret_5d", "VIX_Price_ret_5d__ret5x__LMT_LockheedMartin_ret_1d", "heston_var_ev_h7", "VIX_Price_ret_5d__prod__LMT_LockheedMartin_ret_1d", "ETN_Eaton_ret_1d__macross__LMT_LockheedMartin_ret_1d", "XEL_Xcel_ret_1d__ret5x__DE_Deere_ret_1d", "CMCSA_ret_1d", "heston_var_ev_h3", "CPB_CampbellSoup_ret_5d", "EWJ_Japan_vol_20d", "ENB_EnbridgeInc_ret_1d__ret5x__XEL_Xcel_ret_1d", "vix_vs_ma20__ret5x__VRTX_VertexPharm_ret_1d"], "is_new": false}, {"model_id": "v1_h1_STRESS_LogisticRegression_N20", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 1, "n_features": 20, "F1_dir": 0.5279, "F1_UP_FORT": 0.2276, "F1_DOWN_FORT": 0.5314, "train_start": "2001-02-08", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["ENB_EnbridgeInc_ret_1d__macross__TXN_ret_20d", "ETN_Eaton_ret_1d__div__US6M_Rate_ret_1d", "egarch_spx_delta_h1__macross__VRTX_VertexPharm_ret_1d", "egarch_spx_delta_h1__div__LMT_LockheedMartin_ret_1d", "EXC_Exelon_zscore_60d", "vix_mean_abs_ret_5d", "LMT_LockheedMartin_ret_1d__macross__MCD_ret_1d", "BLK_BlackRock_zscore_60d__ret5x__AMD_ret_5d", "VIX_Price_ret_5d__ret5x__LMT_LockheedMartin_ret_1d", "heston_var_ev_h7", "VIX_Price_ret_5d__prod__LMT_LockheedMartin_ret_1d", "ETN_Eaton_ret_1d__macross__LMT_LockheedMartin_ret_1d", "XEL_Xcel_ret_1d__ret5x__DE_Deere_ret_1d", "CMCSA_ret_1d", "heston_var_ev_h3", "CPB_CampbellSoup_ret_5d", "EWJ_Japan_vol_20d", "ENB_EnbridgeInc_ret_1d__ret5x__XEL_Xcel_ret_1d", "vix_vs_ma20__ret5x__VRTX_VertexPharm_ret_1d", "PPL_PPL_ret_1d__div__BLK_BlackRock_zscore_60d"], "is_new": false}, {"model_id": "v1_h1_STRESS_LogisticRegression_N21", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 1, "n_features": 21, "F1_dir": 0.5162, "F1_UP_FORT": 0.2188, "F1_DOWN_FORT": 0.5019, "train_start": "2001-02-08", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["ENB_EnbridgeInc_ret_1d__macross__TXN_ret_20d", "ETN_Eaton_ret_1d__div__US6M_Rate_ret_1d", "egarch_spx_delta_h1__macross__VRTX_VertexPharm_ret_1d", "egarch_spx_delta_h1__div__LMT_LockheedMartin_ret_1d", "EXC_Exelon_zscore_60d", "vix_mean_abs_ret_5d", "LMT_LockheedMartin_ret_1d__macross__MCD_ret_1d", "BLK_BlackRock_zscore_60d__ret5x__AMD_ret_5d", "VIX_Price_ret_5d__ret5x__LMT_LockheedMartin_ret_1d", "heston_var_ev_h7", "VIX_Price_ret_5d__prod__LMT_LockheedMartin_ret_1d", "ETN_Eaton_ret_1d__macross__LMT_LockheedMartin_ret_1d", "XEL_Xcel_ret_1d__ret5x__DE_Deere_ret_1d", "CMCSA_ret_1d", "heston_var_ev_h3", "CPB_CampbellSoup_ret_5d", "EWJ_Japan_vol_20d", "ENB_EnbridgeInc_ret_1d__ret5x__XEL_Xcel_ret_1d", "vix_vs_ma20__ret5x__VRTX_VertexPharm_ret_1d", "PPL_PPL_ret_1d__div__BLK_BlackRock_zscore_60d", "NFCI_zscore_60d__macross__VRTX_VertexPharm_ret_1d"], "is_new": false}, {"model_id": "v1_h1_STRESS_LogisticRegression_N22", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 1, "n_features": 22, "F1_dir": 0.5225, "F1_UP_FORT": 0.2381, "F1_DOWN_FORT": 0.4925, "train_start": "2001-02-08", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["ENB_EnbridgeInc_ret_1d__macross__TXN_ret_20d", "ETN_Eaton_ret_1d__div__US6M_Rate_ret_1d", "egarch_spx_delta_h1__macross__VRTX_VertexPharm_ret_1d", "egarch_spx_delta_h1__div__LMT_LockheedMartin_ret_1d", "EXC_Exelon_zscore_60d", "vix_mean_abs_ret_5d", "LMT_LockheedMartin_ret_1d__macross__MCD_ret_1d", "BLK_BlackRock_zscore_60d__ret5x__AMD_ret_5d", "VIX_Price_ret_5d__ret5x__LMT_LockheedMartin_ret_1d", "heston_var_ev_h7", "VIX_Price_ret_5d__prod__LMT_LockheedMartin_ret_1d", "ETN_Eaton_ret_1d__macross__LMT_LockheedMartin_ret_1d", "XEL_Xcel_ret_1d__ret5x__DE_Deere_ret_1d", "CMCSA_ret_1d", "heston_var_ev_h3", "CPB_CampbellSoup_ret_5d", "EWJ_Japan_vol_20d", "ENB_EnbridgeInc_ret_1d__ret5x__XEL_Xcel_ret_1d", "vix_vs_ma20__ret5x__VRTX_VertexPharm_ret_1d", "PPL_PPL_ret_1d__div__BLK_BlackRock_zscore_60d", "NFCI_zscore_60d__macross__VRTX_VertexPharm_ret_1d", "PCAR_PaccarInc_ret_5d"], "is_new": false}, {"model_id": "v1_h1_STRESS_LogisticRegression_Optuna_N14", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 1, "n_features": 14, "F1_dir": 0.5342, "F1_UP_FORT": 0.2734, "F1_DOWN_FORT": 0.5132, "train_start": "2001-02-08", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["ENB_EnbridgeInc_ret_1d__macross__TXN_ret_20d", "ETN_Eaton_ret_1d__div__US6M_Rate_ret_1d", "egarch_spx_delta_h1__macross__VRTX_VertexPharm_ret_1d", "egarch_spx_delta_h1__div__LMT_LockheedMartin_ret_1d", "EXC_Exelon_zscore_60d", "vix_mean_abs_ret_5d", "LMT_LockheedMartin_ret_1d__macross__MCD_ret_1d", "BLK_BlackRock_zscore_60d__ret5x__AMD_ret_5d", "VIX_Price_ret_5d__ret5x__LMT_LockheedMartin_ret_1d", "heston_var_ev_h7", "VIX_Price_ret_5d__prod__LMT_LockheedMartin_ret_1d", "ETN_Eaton_ret_1d__macross__LMT_LockheedMartin_ret_1d", "XEL_Xcel_ret_1d__ret5x__DE_Deere_ret_1d", "CMCSA_ret_1d"], "is_new": false}, {"model_id": "v1_h1_GLOBAL_XGBoost_N5", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 1, "n_features": 5, "F1_dir": 0.5305, "F1_UP_FORT": 0.2283, "F1_DOWN_FORT": 0.3936, "train_start": "2000-11-10", "sampler": "SMOTETomek", "best_params": "{}", "features": ["NFCI_ret_5d__div__IWM_SmallCap_vol_20d", "SPY_zscore_60d", "heston_xi__prod__vix_mean_abs_ret_5d", "SJM_JM_Smucker_ret_1d", "vix_momentum_3d__zrel__VZ_ret_5d"], "is_new": false}, {"model_id": "v1_h1_GLOBAL_XGBoost_N6", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 1, "n_features": 6, "F1_dir": 0.5385, "F1_UP_FORT": 0.2648, "F1_DOWN_FORT": 0.4169, "train_start": "2000-11-10", "sampler": "SMOTETomek", "best_params": "{}", "features": ["NFCI_ret_5d__div__IWM_SmallCap_vol_20d", "SPY_zscore_60d", "heston_xi__prod__vix_mean_abs_ret_5d", "SJM_JM_Smucker_ret_1d", "vix_momentum_3d__zrel__VZ_ret_5d", "SBUX_zscore_60d"], "is_new": false}, {"model_id": "v1_h1_GLOBAL_XGBoost_N7", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 1, "n_features": 7, "F1_dir": 0.5413, "F1_UP_FORT": 0.2393, "F1_DOWN_FORT": 0.3966, "train_start": "2000-11-10", "sampler": "SMOTETomek", "best_params": "{}", "features": ["NFCI_ret_5d__div__IWM_SmallCap_vol_20d", "SPY_zscore_60d", "heston_xi__prod__vix_mean_abs_ret_5d", "SJM_JM_Smucker_ret_1d", "vix_momentum_3d__zrel__VZ_ret_5d", "SBUX_zscore_60d", "IWM_SmallCap_vol_20d"], "is_new": false}, {"model_id": "v1_h1_GLOBAL_XGBoost_N8", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 1, "n_features": 8, "F1_dir": 0.5449, "F1_UP_FORT": 0.2218, "F1_DOWN_FORT": 0.3924, "train_start": "2000-11-10", "sampler": "SMOTETomek", "best_params": "{}", "features": ["NFCI_ret_5d__div__IWM_SmallCap_vol_20d", "SPY_zscore_60d", "heston_xi__prod__vix_mean_abs_ret_5d", "SJM_JM_Smucker_ret_1d", "vix_momentum_3d__zrel__VZ_ret_5d", "SBUX_zscore_60d", "IWM_SmallCap_vol_20d", "SLB_Schlumberger_ret_5d"], "is_new": false}, {"model_id": "v1_h1_GLOBAL_XGBoost_N9", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 1, "n_features": 9, "F1_dir": 0.5451, "F1_UP_FORT": 0.2162, "F1_DOWN_FORT": 0.3961, "train_start": "2000-11-10", "sampler": "SMOTETomek", "best_params": "{}", "features": ["NFCI_ret_5d__div__IWM_SmallCap_vol_20d", "SPY_zscore_60d", "heston_xi__prod__vix_mean_abs_ret_5d", "SJM_JM_Smucker_ret_1d", "vix_momentum_3d__zrel__VZ_ret_5d", "SBUX_zscore_60d", "IWM_SmallCap_vol_20d", "SLB_Schlumberger_ret_5d", "STLFSI4_zscore_60d__ret5x__PPL_PPL_ret_5d"], "is_new": false}, {"model_id": "v1_h1_GLOBAL_XGBoost_N10", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 1, "n_features": 10, "F1_dir": 0.5545, "F1_UP_FORT": 0.2505, "F1_DOWN_FORT": 0.4228, "train_start": "2000-11-10", "sampler": "SMOTETomek", "best_params": "{}", "features": ["NFCI_ret_5d__div__IWM_SmallCap_vol_20d", "SPY_zscore_60d", "heston_xi__prod__vix_mean_abs_ret_5d", "SJM_JM_Smucker_ret_1d", "vix_momentum_3d__zrel__VZ_ret_5d", "SBUX_zscore_60d", "IWM_SmallCap_vol_20d", "SLB_Schlumberger_ret_5d", "STLFSI4_zscore_60d__ret5x__PPL_PPL_ret_5d", "EBAY_eBay_ret_1d__div__vix_mean_abs_ret_5d"], "is_new": false}, {"model_id": "v1_h1_GLOBAL_XGBoost_N11", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 1, "n_features": 11, "F1_dir": 0.555, "F1_UP_FORT": 0.2457, "F1_DOWN_FORT": 0.4179, "train_start": "2000-11-10", "sampler": "SMOTETomek", "best_params": "{}", "features": ["NFCI_ret_5d__div__IWM_SmallCap_vol_20d", "SPY_zscore_60d", "heston_xi__prod__vix_mean_abs_ret_5d", "SJM_JM_Smucker_ret_1d", "vix_momentum_3d__zrel__VZ_ret_5d", "SBUX_zscore_60d", "IWM_SmallCap_vol_20d", "SLB_Schlumberger_ret_5d", "STLFSI4_zscore_60d__ret5x__PPL_PPL_ret_5d", "EBAY_eBay_ret_1d__div__vix_mean_abs_ret_5d", "heston_xi__minus__EBAY_eBay_ret_1d"], "is_new": false}, {"model_id": "v1_h1_GLOBAL_XGBoost_N12", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 1, "n_features": 12, "F1_dir": 0.541, "F1_UP_FORT": 0.2299, "F1_DOWN_FORT": 0.4026, "train_start": "2000-11-10", "sampler": "SMOTETomek", "best_params": "{}", "features": ["NFCI_ret_5d__div__IWM_SmallCap_vol_20d", "SPY_zscore_60d", "heston_xi__prod__vix_mean_abs_ret_5d", "SJM_JM_Smucker_ret_1d", "vix_momentum_3d__zrel__VZ_ret_5d", "SBUX_zscore_60d", "IWM_SmallCap_vol_20d", "SLB_Schlumberger_ret_5d", "STLFSI4_zscore_60d__ret5x__PPL_PPL_ret_5d", "EBAY_eBay_ret_1d__div__vix_mean_abs_ret_5d", "heston_xi__minus__EBAY_eBay_ret_1d", "NFCI_ret_5d__div__vix_mean_abs_ret_5d"], "is_new": false}, {"model_id": "v1_h1_GLOBAL_XGBoost_N13", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 1, "n_features": 13, "F1_dir": 0.5382, "F1_UP_FORT": 0.2206, "F1_DOWN_FORT": 0.4208, "train_start": "2000-11-10", "sampler": "SMOTETomek", "best_params": "{}", "features": ["NFCI_ret_5d__div__IWM_SmallCap_vol_20d", "SPY_zscore_60d", "heston_xi__prod__vix_mean_abs_ret_5d", "SJM_JM_Smucker_ret_1d", "vix_momentum_3d__zrel__VZ_ret_5d", "SBUX_zscore_60d", "IWM_SmallCap_vol_20d", "SLB_Schlumberger_ret_5d", "STLFSI4_zscore_60d__ret5x__PPL_PPL_ret_5d", "EBAY_eBay_ret_1d__div__vix_mean_abs_ret_5d", "heston_xi__minus__EBAY_eBay_ret_1d", "NFCI_ret_5d__div__vix_mean_abs_ret_5d", "EXC_Exelon_ret_1d"], "is_new": false}, {"model_id": "v1_h1_GLOBAL_XGBoost_N14", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 1, "n_features": 14, "F1_dir": 0.5346, "F1_UP_FORT": 0.225, "F1_DOWN_FORT": 0.4223, "train_start": "2000-11-10", "sampler": "SMOTETomek", "best_params": "{}", "features": ["NFCI_ret_5d__div__IWM_SmallCap_vol_20d", "SPY_zscore_60d", "heston_xi__prod__vix_mean_abs_ret_5d", "SJM_JM_Smucker_ret_1d", "vix_momentum_3d__zrel__VZ_ret_5d", "SBUX_zscore_60d", "IWM_SmallCap_vol_20d", "SLB_Schlumberger_ret_5d", "STLFSI4_zscore_60d__ret5x__PPL_PPL_ret_5d", "EBAY_eBay_ret_1d__div__vix_mean_abs_ret_5d", "heston_xi__minus__EBAY_eBay_ret_1d", "NFCI_ret_5d__div__vix_mean_abs_ret_5d", "EXC_Exelon_ret_1d", "EBAY_eBay_ret_1d__ret5x__PPL_PPL_ret_5d"], "is_new": false}, {"model_id": "v1_h1_GLOBAL_XGBoost_N15", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 1, "n_features": 15, "F1_dir": 0.5398, "F1_UP_FORT": 0.2132, "F1_DOWN_FORT": 0.4149, "train_start": "2000-11-10", "sampler": "SMOTETomek", "best_params": "{}", "features": ["NFCI_ret_5d__div__IWM_SmallCap_vol_20d", "SPY_zscore_60d", "heston_xi__prod__vix_mean_abs_ret_5d", "SJM_JM_Smucker_ret_1d", "vix_momentum_3d__zrel__VZ_ret_5d", "SBUX_zscore_60d", "IWM_SmallCap_vol_20d", "SLB_Schlumberger_ret_5d", "STLFSI4_zscore_60d__ret5x__PPL_PPL_ret_5d", "EBAY_eBay_ret_1d__div__vix_mean_abs_ret_5d", "heston_xi__minus__EBAY_eBay_ret_1d", "NFCI_ret_5d__div__vix_mean_abs_ret_5d", "EXC_Exelon_ret_1d", "EBAY_eBay_ret_1d__ret5x__PPL_PPL_ret_5d", "NFCI_ret_5d__ret5x__vix_momentum_3d"], "is_new": false}, {"model_id": "v1_h1_GLOBAL_XGBoost_N16", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 1, "n_features": 16, "F1_dir": 0.5467, "F1_UP_FORT": 0.2361, "F1_DOWN_FORT": 0.4159, "train_start": "2000-11-10", "sampler": "SMOTETomek", "best_params": "{}", "features": ["NFCI_ret_5d__div__IWM_SmallCap_vol_20d", "SPY_zscore_60d", "heston_xi__prod__vix_mean_abs_ret_5d", "SJM_JM_Smucker_ret_1d", "vix_momentum_3d__zrel__VZ_ret_5d", "SBUX_zscore_60d", "IWM_SmallCap_vol_20d", "SLB_Schlumberger_ret_5d", "STLFSI4_zscore_60d__ret5x__PPL_PPL_ret_5d", "EBAY_eBay_ret_1d__div__vix_mean_abs_ret_5d", "heston_xi__minus__EBAY_eBay_ret_1d", "NFCI_ret_5d__div__vix_mean_abs_ret_5d", "EXC_Exelon_ret_1d", "EBAY_eBay_ret_1d__ret5x__PPL_PPL_ret_5d", "NFCI_ret_5d__ret5x__vix_momentum_3d", "heston_xi__div__COST_vol_20d"], "is_new": false}, {"model_id": "v1_h1_GLOBAL_XGBoost_N17", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 1, "n_features": 17, "F1_dir": 0.5447, "F1_UP_FORT": 0.2409, "F1_DOWN_FORT": 0.4338, "train_start": "2000-11-10", "sampler": "SMOTETomek", "best_params": "{}", "features": ["NFCI_ret_5d__div__IWM_SmallCap_vol_20d", "SPY_zscore_60d", "heston_xi__prod__vix_mean_abs_ret_5d", "SJM_JM_Smucker_ret_1d", "vix_momentum_3d__zrel__VZ_ret_5d", "SBUX_zscore_60d", "IWM_SmallCap_vol_20d", "SLB_Schlumberger_ret_5d", "STLFSI4_zscore_60d__ret5x__PPL_PPL_ret_5d", "EBAY_eBay_ret_1d__div__vix_mean_abs_ret_5d", "heston_xi__minus__EBAY_eBay_ret_1d", "NFCI_ret_5d__div__vix_mean_abs_ret_5d", "EXC_Exelon_ret_1d", "EBAY_eBay_ret_1d__ret5x__PPL_PPL_ret_5d", "NFCI_ret_5d__ret5x__vix_momentum_3d", "heston_xi__div__COST_vol_20d", "US1Y_Rate_ret_5d"], "is_new": false}, {"model_id": "v1_h1_GLOBAL_XGBoost_N18", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 1, "n_features": 18, "F1_dir": 0.5377, "F1_UP_FORT": 0.2385, "F1_DOWN_FORT": 0.4158, "train_start": "2000-11-10", "sampler": "SMOTETomek", "best_params": "{}", "features": ["NFCI_ret_5d__div__IWM_SmallCap_vol_20d", "SPY_zscore_60d", "heston_xi__prod__vix_mean_abs_ret_5d", "SJM_JM_Smucker_ret_1d", "vix_momentum_3d__zrel__VZ_ret_5d", "SBUX_zscore_60d", "IWM_SmallCap_vol_20d", "SLB_Schlumberger_ret_5d", "STLFSI4_zscore_60d__ret5x__PPL_PPL_ret_5d", "EBAY_eBay_ret_1d__div__vix_mean_abs_ret_5d", "heston_xi__minus__EBAY_eBay_ret_1d", "NFCI_ret_5d__div__vix_mean_abs_ret_5d", "EXC_Exelon_ret_1d", "EBAY_eBay_ret_1d__ret5x__PPL_PPL_ret_5d", "NFCI_ret_5d__ret5x__vix_momentum_3d", "heston_xi__div__COST_vol_20d", "US1Y_Rate_ret_5d", "IWM_SmallCap_vol_20d__div__BAC_ret_1d"], "is_new": false}, {"model_id": "v1_h1_GLOBAL_XGBoost_N19", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 1, "n_features": 19, "F1_dir": 0.5369, "F1_UP_FORT": 0.2495, "F1_DOWN_FORT": 0.4355, "train_start": "2000-11-10", "sampler": "SMOTETomek", "best_params": "{}", "features": ["NFCI_ret_5d__div__IWM_SmallCap_vol_20d", "SPY_zscore_60d", "heston_xi__prod__vix_mean_abs_ret_5d", "SJM_JM_Smucker_ret_1d", "vix_momentum_3d__zrel__VZ_ret_5d", "SBUX_zscore_60d", "IWM_SmallCap_vol_20d", "SLB_Schlumberger_ret_5d", "STLFSI4_zscore_60d__ret5x__PPL_PPL_ret_5d", "EBAY_eBay_ret_1d__div__vix_mean_abs_ret_5d", "heston_xi__minus__EBAY_eBay_ret_1d", "NFCI_ret_5d__div__vix_mean_abs_ret_5d", "EXC_Exelon_ret_1d", "EBAY_eBay_ret_1d__ret5x__PPL_PPL_ret_5d", "NFCI_ret_5d__ret5x__vix_momentum_3d", "heston_xi__div__COST_vol_20d", "US1Y_Rate_ret_5d", "IWM_SmallCap_vol_20d__div__BAC_ret_1d", "ROST_RossStores_ret_1d__minus__COST_vol_20d"], "is_new": false}, {"model_id": "v1_h1_GLOBAL_XGBoost_N20", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 1, "n_features": 20, "F1_dir": 0.5261, "F1_UP_FORT": 0.2131, "F1_DOWN_FORT": 0.4334, "train_start": "2000-11-10", "sampler": "SMOTETomek", "best_params": "{}", "features": ["NFCI_ret_5d__div__IWM_SmallCap_vol_20d", "SPY_zscore_60d", "heston_xi__prod__vix_mean_abs_ret_5d", "SJM_JM_Smucker_ret_1d", "vix_momentum_3d__zrel__VZ_ret_5d", "SBUX_zscore_60d", "IWM_SmallCap_vol_20d", "SLB_Schlumberger_ret_5d", "STLFSI4_zscore_60d__ret5x__PPL_PPL_ret_5d", "EBAY_eBay_ret_1d__div__vix_mean_abs_ret_5d", "heston_xi__minus__EBAY_eBay_ret_1d", "NFCI_ret_5d__div__vix_mean_abs_ret_5d", "EXC_Exelon_ret_1d", "EBAY_eBay_ret_1d__ret5x__PPL_PPL_ret_5d", "NFCI_ret_5d__ret5x__vix_momentum_3d", "heston_xi__div__COST_vol_20d", "US1Y_Rate_ret_5d", "IWM_SmallCap_vol_20d__div__BAC_ret_1d", "ROST_RossStores_ret_1d__minus__COST_vol_20d", "STLFSI4_zscore_60d__ret5x__vix_zscore_10d"], "is_new": false}, {"model_id": "v1_h1_GLOBAL_LightGBM_N5", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 1, "n_features": 5, "F1_dir": 0.5259, "F1_UP_FORT": 0.243, "F1_DOWN_FORT": 0.3812, "train_start": "2000-11-10", "sampler": "SMOTETomek", "best_params": "{}", "features": ["NFCI_ret_5d__div__IWM_SmallCap_vol_20d", "SPY_zscore_60d", "heston_xi__prod__vix_mean_abs_ret_5d", "SJM_JM_Smucker_ret_1d", "vix_momentum_3d__zrel__VZ_ret_5d"], "is_new": false}, {"model_id": "v1_h1_GLOBAL_LightGBM_N6", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 1, "n_features": 6, "F1_dir": 0.5237, "F1_UP_FORT": 0.2276, "F1_DOWN_FORT": 0.4125, "train_start": "2000-11-10", "sampler": "SMOTETomek", "best_params": "{}", "features": ["NFCI_ret_5d__div__IWM_SmallCap_vol_20d", "SPY_zscore_60d", "heston_xi__prod__vix_mean_abs_ret_5d", "SJM_JM_Smucker_ret_1d", "vix_momentum_3d__zrel__VZ_ret_5d", "SBUX_zscore_60d"], "is_new": false}, {"model_id": "v1_h1_GLOBAL_LightGBM_N7", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 1, "n_features": 7, "F1_dir": 0.5341, "F1_UP_FORT": 0.256, "F1_DOWN_FORT": 0.3835, "train_start": "2000-11-10", "sampler": "SMOTETomek", "best_params": "{}", "features": ["NFCI_ret_5d__div__IWM_SmallCap_vol_20d", "SPY_zscore_60d", "heston_xi__prod__vix_mean_abs_ret_5d", "SJM_JM_Smucker_ret_1d", "vix_momentum_3d__zrel__VZ_ret_5d", "SBUX_zscore_60d", "IWM_SmallCap_vol_20d"], "is_new": false}, {"model_id": "v1_h1_GLOBAL_LightGBM_N8", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 1, "n_features": 8, "F1_dir": 0.5317, "F1_UP_FORT": 0.2274, "F1_DOWN_FORT": 0.394, "train_start": "2000-11-10", "sampler": "SMOTETomek", "best_params": "{}", "features": ["NFCI_ret_5d__div__IWM_SmallCap_vol_20d", "SPY_zscore_60d", "heston_xi__prod__vix_mean_abs_ret_5d", "SJM_JM_Smucker_ret_1d", "vix_momentum_3d__zrel__VZ_ret_5d", "SBUX_zscore_60d", "IWM_SmallCap_vol_20d", "SLB_Schlumberger_ret_5d"], "is_new": false}, {"model_id": "v1_h1_GLOBAL_LightGBM_N9", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 1, "n_features": 9, "F1_dir": 0.5322, "F1_UP_FORT": 0.2205, "F1_DOWN_FORT": 0.3824, "train_start": "2000-11-10", "sampler": "SMOTETomek", "best_params": "{}", "features": ["NFCI_ret_5d__div__IWM_SmallCap_vol_20d", "SPY_zscore_60d", "heston_xi__prod__vix_mean_abs_ret_5d", "SJM_JM_Smucker_ret_1d", "vix_momentum_3d__zrel__VZ_ret_5d", "SBUX_zscore_60d", "IWM_SmallCap_vol_20d", "SLB_Schlumberger_ret_5d", "STLFSI4_zscore_60d__ret5x__PPL_PPL_ret_5d"], "is_new": false}, {"model_id": "v1_h1_GLOBAL_LightGBM_N10", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 1, "n_features": 10, "F1_dir": 0.535, "F1_UP_FORT": 0.2326, "F1_DOWN_FORT": 0.3729, "train_start": "2000-11-10", "sampler": "SMOTETomek", "best_params": "{}", "features": ["NFCI_ret_5d__div__IWM_SmallCap_vol_20d", "SPY_zscore_60d", "heston_xi__prod__vix_mean_abs_ret_5d", "SJM_JM_Smucker_ret_1d", "vix_momentum_3d__zrel__VZ_ret_5d", "SBUX_zscore_60d", "IWM_SmallCap_vol_20d", "SLB_Schlumberger_ret_5d", "STLFSI4_zscore_60d__ret5x__PPL_PPL_ret_5d", "EBAY_eBay_ret_1d__div__vix_mean_abs_ret_5d"], "is_new": false}, {"model_id": "v1_h1_GLOBAL_LightGBM_N11", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 1, "n_features": 11, "F1_dir": 0.5492, "F1_UP_FORT": 0.25, "F1_DOWN_FORT": 0.3874, "train_start": "2000-11-10", "sampler": "SMOTETomek", "best_params": "{}", "features": ["NFCI_ret_5d__div__IWM_SmallCap_vol_20d", "SPY_zscore_60d", "heston_xi__prod__vix_mean_abs_ret_5d", "SJM_JM_Smucker_ret_1d", "vix_momentum_3d__zrel__VZ_ret_5d", "SBUX_zscore_60d", "IWM_SmallCap_vol_20d", "SLB_Schlumberger_ret_5d", "STLFSI4_zscore_60d__ret5x__PPL_PPL_ret_5d", "EBAY_eBay_ret_1d__div__vix_mean_abs_ret_5d", "heston_xi__minus__EBAY_eBay_ret_1d"], "is_new": false}, {"model_id": "v1_h1_GLOBAL_LightGBM_N12", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 1, "n_features": 12, "F1_dir": 0.5333, "F1_UP_FORT": 0.2646, "F1_DOWN_FORT": 0.4064, "train_start": "2000-11-10", "sampler": "SMOTETomek", "best_params": "{}", "features": ["NFCI_ret_5d__div__IWM_SmallCap_vol_20d", "SPY_zscore_60d", "heston_xi__prod__vix_mean_abs_ret_5d", "SJM_JM_Smucker_ret_1d", "vix_momentum_3d__zrel__VZ_ret_5d", "SBUX_zscore_60d", "IWM_SmallCap_vol_20d", "SLB_Schlumberger_ret_5d", "STLFSI4_zscore_60d__ret5x__PPL_PPL_ret_5d", "EBAY_eBay_ret_1d__div__vix_mean_abs_ret_5d", "heston_xi__minus__EBAY_eBay_ret_1d", "NFCI_ret_5d__div__vix_mean_abs_ret_5d"], "is_new": false}, {"model_id": "v1_h1_GLOBAL_LightGBM_N13", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 1, "n_features": 13, "F1_dir": 0.5315, "F1_UP_FORT": 0.2323, "F1_DOWN_FORT": 0.3871, "train_start": "2000-11-10", "sampler": "SMOTETomek", "best_params": "{}", "features": ["NFCI_ret_5d__div__IWM_SmallCap_vol_20d", "SPY_zscore_60d", "heston_xi__prod__vix_mean_abs_ret_5d", "SJM_JM_Smucker_ret_1d", "vix_momentum_3d__zrel__VZ_ret_5d", "SBUX_zscore_60d", "IWM_SmallCap_vol_20d", "SLB_Schlumberger_ret_5d", "STLFSI4_zscore_60d__ret5x__PPL_PPL_ret_5d", "EBAY_eBay_ret_1d__div__vix_mean_abs_ret_5d", "heston_xi__minus__EBAY_eBay_ret_1d", "NFCI_ret_5d__div__vix_mean_abs_ret_5d", "EXC_Exelon_ret_1d"], "is_new": false}, {"model_id": "v1_h1_GLOBAL_LightGBM_N14", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 1, "n_features": 14, "F1_dir": 0.5174, "F1_UP_FORT": 0.2483, "F1_DOWN_FORT": 0.4069, "train_start": "2000-11-10", "sampler": "SMOTETomek", "best_params": "{}", "features": ["NFCI_ret_5d__div__IWM_SmallCap_vol_20d", "SPY_zscore_60d", "heston_xi__prod__vix_mean_abs_ret_5d", "SJM_JM_Smucker_ret_1d", "vix_momentum_3d__zrel__VZ_ret_5d", "SBUX_zscore_60d", "IWM_SmallCap_vol_20d", "SLB_Schlumberger_ret_5d", "STLFSI4_zscore_60d__ret5x__PPL_PPL_ret_5d", "EBAY_eBay_ret_1d__div__vix_mean_abs_ret_5d", "heston_xi__minus__EBAY_eBay_ret_1d", "NFCI_ret_5d__div__vix_mean_abs_ret_5d", "EXC_Exelon_ret_1d", "EBAY_eBay_ret_1d__ret5x__PPL_PPL_ret_5d"], "is_new": false}, {"model_id": "v1_h1_GLOBAL_LightGBM_N15", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 1, "n_features": 15, "F1_dir": 0.5447, "F1_UP_FORT": 0.2606, "F1_DOWN_FORT": 0.3984, "train_start": "2000-11-10", "sampler": "SMOTETomek", "best_params": "{}", "features": ["NFCI_ret_5d__div__IWM_SmallCap_vol_20d", "SPY_zscore_60d", "heston_xi__prod__vix_mean_abs_ret_5d", "SJM_JM_Smucker_ret_1d", "vix_momentum_3d__zrel__VZ_ret_5d", "SBUX_zscore_60d", "IWM_SmallCap_vol_20d", "SLB_Schlumberger_ret_5d", "STLFSI4_zscore_60d__ret5x__PPL_PPL_ret_5d", "EBAY_eBay_ret_1d__div__vix_mean_abs_ret_5d", "heston_xi__minus__EBAY_eBay_ret_1d", "NFCI_ret_5d__div__vix_mean_abs_ret_5d", "EXC_Exelon_ret_1d", "EBAY_eBay_ret_1d__ret5x__PPL_PPL_ret_5d", "NFCI_ret_5d__ret5x__vix_momentum_3d"], "is_new": false}, {"model_id": "v1_h1_GLOBAL_LightGBM_N16", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 1, "n_features": 16, "F1_dir": 0.5453, "F1_UP_FORT": 0.2887, "F1_DOWN_FORT": 0.4172, "train_start": "2000-11-10", "sampler": "SMOTETomek", "best_params": "{}", "features": ["NFCI_ret_5d__div__IWM_SmallCap_vol_20d", "SPY_zscore_60d", "heston_xi__prod__vix_mean_abs_ret_5d", "SJM_JM_Smucker_ret_1d", "vix_momentum_3d__zrel__VZ_ret_5d", "SBUX_zscore_60d", "IWM_SmallCap_vol_20d", "SLB_Schlumberger_ret_5d", "STLFSI4_zscore_60d__ret5x__PPL_PPL_ret_5d", "EBAY_eBay_ret_1d__div__vix_mean_abs_ret_5d", "heston_xi__minus__EBAY_eBay_ret_1d", "NFCI_ret_5d__div__vix_mean_abs_ret_5d", "EXC_Exelon_ret_1d", "EBAY_eBay_ret_1d__ret5x__PPL_PPL_ret_5d", "NFCI_ret_5d__ret5x__vix_momentum_3d", "heston_xi__div__COST_vol_20d"], "is_new": false}, {"model_id": "v1_h1_GLOBAL_LightGBM_N17", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 1, "n_features": 17, "F1_dir": 0.5404, "F1_UP_FORT": 0.2451, "F1_DOWN_FORT": 0.4199, "train_start": "2000-11-10", "sampler": "SMOTETomek", "best_params": "{}", "features": ["NFCI_ret_5d__div__IWM_SmallCap_vol_20d", "SPY_zscore_60d", "heston_xi__prod__vix_mean_abs_ret_5d", "SJM_JM_Smucker_ret_1d", "vix_momentum_3d__zrel__VZ_ret_5d", "SBUX_zscore_60d", "IWM_SmallCap_vol_20d", "SLB_Schlumberger_ret_5d", "STLFSI4_zscore_60d__ret5x__PPL_PPL_ret_5d", "EBAY_eBay_ret_1d__div__vix_mean_abs_ret_5d", "heston_xi__minus__EBAY_eBay_ret_1d", "NFCI_ret_5d__div__vix_mean_abs_ret_5d", "EXC_Exelon_ret_1d", "EBAY_eBay_ret_1d__ret5x__PPL_PPL_ret_5d", "NFCI_ret_5d__ret5x__vix_momentum_3d", "heston_xi__div__COST_vol_20d", "US1Y_Rate_ret_5d"], "is_new": false}, {"model_id": "v1_h1_GLOBAL_LightGBM_N18", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 1, "n_features": 18, "F1_dir": 0.5338, "F1_UP_FORT": 0.2487, "F1_DOWN_FORT": 0.4054, "train_start": "2000-11-10", "sampler": "SMOTETomek", "best_params": "{}", "features": ["NFCI_ret_5d__div__IWM_SmallCap_vol_20d", "SPY_zscore_60d", "heston_xi__prod__vix_mean_abs_ret_5d", "SJM_JM_Smucker_ret_1d", "vix_momentum_3d__zrel__VZ_ret_5d", "SBUX_zscore_60d", "IWM_SmallCap_vol_20d", "SLB_Schlumberger_ret_5d", "STLFSI4_zscore_60d__ret5x__PPL_PPL_ret_5d", "EBAY_eBay_ret_1d__div__vix_mean_abs_ret_5d", "heston_xi__minus__EBAY_eBay_ret_1d", "NFCI_ret_5d__div__vix_mean_abs_ret_5d", "EXC_Exelon_ret_1d", "EBAY_eBay_ret_1d__ret5x__PPL_PPL_ret_5d", "NFCI_ret_5d__ret5x__vix_momentum_3d", "heston_xi__div__COST_vol_20d", "US1Y_Rate_ret_5d", "IWM_SmallCap_vol_20d__div__BAC_ret_1d"], "is_new": false}, {"model_id": "v1_h1_GLOBAL_LightGBM_N19", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 1, "n_features": 19, "F1_dir": 0.5257, "F1_UP_FORT": 0.2705, "F1_DOWN_FORT": 0.4164, "train_start": "2000-11-10", "sampler": "SMOTETomek", "best_params": "{}", "features": ["NFCI_ret_5d__div__IWM_SmallCap_vol_20d", "SPY_zscore_60d", "heston_xi__prod__vix_mean_abs_ret_5d", "SJM_JM_Smucker_ret_1d", "vix_momentum_3d__zrel__VZ_ret_5d", "SBUX_zscore_60d", "IWM_SmallCap_vol_20d", "SLB_Schlumberger_ret_5d", "STLFSI4_zscore_60d__ret5x__PPL_PPL_ret_5d", "EBAY_eBay_ret_1d__div__vix_mean_abs_ret_5d", "heston_xi__minus__EBAY_eBay_ret_1d", "NFCI_ret_5d__div__vix_mean_abs_ret_5d", "EXC_Exelon_ret_1d", "EBAY_eBay_ret_1d__ret5x__PPL_PPL_ret_5d", "NFCI_ret_5d__ret5x__vix_momentum_3d", "heston_xi__div__COST_vol_20d", "US1Y_Rate_ret_5d", "IWM_SmallCap_vol_20d__div__BAC_ret_1d", "ROST_RossStores_ret_1d__minus__COST_vol_20d"], "is_new": false}, {"model_id": "v1_h1_GLOBAL_LightGBM_N20", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 1, "n_features": 20, "F1_dir": 0.5345, "F1_UP_FORT": 0.2509, "F1_DOWN_FORT": 0.4149, "train_start": "2000-11-10", "sampler": "SMOTETomek", "best_params": "{}", "features": ["NFCI_ret_5d__div__IWM_SmallCap_vol_20d", "SPY_zscore_60d", "heston_xi__prod__vix_mean_abs_ret_5d", "SJM_JM_Smucker_ret_1d", "vix_momentum_3d__zrel__VZ_ret_5d", "SBUX_zscore_60d", "IWM_SmallCap_vol_20d", "SLB_Schlumberger_ret_5d", "STLFSI4_zscore_60d__ret5x__PPL_PPL_ret_5d", "EBAY_eBay_ret_1d__div__vix_mean_abs_ret_5d", "heston_xi__minus__EBAY_eBay_ret_1d", "NFCI_ret_5d__div__vix_mean_abs_ret_5d", "EXC_Exelon_ret_1d", "EBAY_eBay_ret_1d__ret5x__PPL_PPL_ret_5d", "NFCI_ret_5d__ret5x__vix_momentum_3d", "heston_xi__div__COST_vol_20d", "US1Y_Rate_ret_5d", "IWM_SmallCap_vol_20d__div__BAC_ret_1d", "ROST_RossStores_ret_1d__minus__COST_vol_20d", "STLFSI4_zscore_60d__ret5x__vix_zscore_10d"], "is_new": false}, {"model_id": "v1_h1_GLOBAL_GradientBoosting_N5", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 1, "n_features": 5, "F1_dir": 0.5563, "F1_UP_FORT": 0.2206, "F1_DOWN_FORT": 0.4146, "train_start": "2000-11-10", "sampler": "SMOTETomek", "best_params": "{}", "features": ["NFCI_ret_5d__div__IWM_SmallCap_vol_20d", "SPY_zscore_60d", "heston_xi__prod__vix_mean_abs_ret_5d", "SJM_JM_Smucker_ret_1d", "vix_momentum_3d__zrel__VZ_ret_5d"], "is_new": false}, {"model_id": "v1_h1_GLOBAL_GradientBoosting_N6", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 1, "n_features": 6, "F1_dir": 0.5424, "F1_UP_FORT": 0.2447, "F1_DOWN_FORT": 0.4, "train_start": "2000-11-10", "sampler": "SMOTETomek", "best_params": "{}", "features": ["NFCI_ret_5d__div__IWM_SmallCap_vol_20d", "SPY_zscore_60d", "heston_xi__prod__vix_mean_abs_ret_5d", "SJM_JM_Smucker_ret_1d", "vix_momentum_3d__zrel__VZ_ret_5d", "SBUX_zscore_60d"], "is_new": false}, {"model_id": "v1_h1_GLOBAL_GradientBoosting_N7", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 1, "n_features": 7, "F1_dir": 0.553, "F1_UP_FORT": 0.2751, "F1_DOWN_FORT": 0.4017, "train_start": "2000-11-10", "sampler": "SMOTETomek", "best_params": "{}", "features": ["NFCI_ret_5d__div__IWM_SmallCap_vol_20d", "SPY_zscore_60d", "heston_xi__prod__vix_mean_abs_ret_5d", "SJM_JM_Smucker_ret_1d", "vix_momentum_3d__zrel__VZ_ret_5d", "SBUX_zscore_60d", "IWM_SmallCap_vol_20d"], "is_new": false}, {"model_id": "v1_h1_GLOBAL_GradientBoosting_N8", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 1, "n_features": 8, "F1_dir": 0.5357, "F1_UP_FORT": 0.2004, "F1_DOWN_FORT": 0.4063, "train_start": "2000-11-10", "sampler": "SMOTETomek", "best_params": "{}", "features": ["NFCI_ret_5d__div__IWM_SmallCap_vol_20d", "SPY_zscore_60d", "heston_xi__prod__vix_mean_abs_ret_5d", "SJM_JM_Smucker_ret_1d", "vix_momentum_3d__zrel__VZ_ret_5d", "SBUX_zscore_60d", "IWM_SmallCap_vol_20d", "SLB_Schlumberger_ret_5d"], "is_new": false}, {"model_id": "v1_h1_GLOBAL_GradientBoosting_N9", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 1, "n_features": 9, "F1_dir": 0.5361, "F1_UP_FORT": 0.244, "F1_DOWN_FORT": 0.3746, "train_start": "2000-11-10", "sampler": "SMOTETomek", "best_params": "{}", "features": ["NFCI_ret_5d__div__IWM_SmallCap_vol_20d", "SPY_zscore_60d", "heston_xi__prod__vix_mean_abs_ret_5d", "SJM_JM_Smucker_ret_1d", "vix_momentum_3d__zrel__VZ_ret_5d", "SBUX_zscore_60d", "IWM_SmallCap_vol_20d", "SLB_Schlumberger_ret_5d", "STLFSI4_zscore_60d__ret5x__PPL_PPL_ret_5d"], "is_new": false}, {"model_id": "v1_h1_GLOBAL_GradientBoosting_N10", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 1, "n_features": 10, "F1_dir": 0.543, "F1_UP_FORT": 0.2004, "F1_DOWN_FORT": 0.3939, "train_start": "2000-11-10", "sampler": "SMOTETomek", "best_params": "{}", "features": ["NFCI_ret_5d__div__IWM_SmallCap_vol_20d", "SPY_zscore_60d", "heston_xi__prod__vix_mean_abs_ret_5d", "SJM_JM_Smucker_ret_1d", "vix_momentum_3d__zrel__VZ_ret_5d", "SBUX_zscore_60d", "IWM_SmallCap_vol_20d", "SLB_Schlumberger_ret_5d", "STLFSI4_zscore_60d__ret5x__PPL_PPL_ret_5d", "EBAY_eBay_ret_1d__div__vix_mean_abs_ret_5d"], "is_new": false}, {"model_id": "v1_h1_GLOBAL_GradientBoosting_N11", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 1, "n_features": 11, "F1_dir": 0.5457, "F1_UP_FORT": 0.24, "F1_DOWN_FORT": 0.4267, "train_start": "2000-11-10", "sampler": "SMOTETomek", "best_params": "{}", "features": ["NFCI_ret_5d__div__IWM_SmallCap_vol_20d", "SPY_zscore_60d", "heston_xi__prod__vix_mean_abs_ret_5d", "SJM_JM_Smucker_ret_1d", "vix_momentum_3d__zrel__VZ_ret_5d", "SBUX_zscore_60d", "IWM_SmallCap_vol_20d", "SLB_Schlumberger_ret_5d", "STLFSI4_zscore_60d__ret5x__PPL_PPL_ret_5d", "EBAY_eBay_ret_1d__div__vix_mean_abs_ret_5d", "heston_xi__minus__EBAY_eBay_ret_1d"], "is_new": false}, {"model_id": "v1_h1_GLOBAL_GradientBoosting_N12", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 1, "n_features": 12, "F1_dir": 0.5504, "F1_UP_FORT": 0.2303, "F1_DOWN_FORT": 0.41, "train_start": "2000-11-10", "sampler": "SMOTETomek", "best_params": "{}", "features": ["NFCI_ret_5d__div__IWM_SmallCap_vol_20d", "SPY_zscore_60d", "heston_xi__prod__vix_mean_abs_ret_5d", "SJM_JM_Smucker_ret_1d", "vix_momentum_3d__zrel__VZ_ret_5d", "SBUX_zscore_60d", "IWM_SmallCap_vol_20d", "SLB_Schlumberger_ret_5d", "STLFSI4_zscore_60d__ret5x__PPL_PPL_ret_5d", "EBAY_eBay_ret_1d__div__vix_mean_abs_ret_5d", "heston_xi__minus__EBAY_eBay_ret_1d", "NFCI_ret_5d__div__vix_mean_abs_ret_5d"], "is_new": false}, {"model_id": "v1_h1_GLOBAL_GradientBoosting_N13", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 1, "n_features": 13, "F1_dir": 0.5221, "F1_UP_FORT": 0.2399, "F1_DOWN_FORT": 0.4005, "train_start": "2000-11-10", "sampler": "SMOTETomek", "best_params": "{}", "features": ["NFCI_ret_5d__div__IWM_SmallCap_vol_20d", "SPY_zscore_60d", "heston_xi__prod__vix_mean_abs_ret_5d", "SJM_JM_Smucker_ret_1d", "vix_momentum_3d__zrel__VZ_ret_5d", "SBUX_zscore_60d", "IWM_SmallCap_vol_20d", "SLB_Schlumberger_ret_5d", "STLFSI4_zscore_60d__ret5x__PPL_PPL_ret_5d", "EBAY_eBay_ret_1d__div__vix_mean_abs_ret_5d", "heston_xi__minus__EBAY_eBay_ret_1d", "NFCI_ret_5d__div__vix_mean_abs_ret_5d", "EXC_Exelon_ret_1d"], "is_new": false}, {"model_id": "v1_h1_GLOBAL_GradientBoosting_N14", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 1, "n_features": 14, "F1_dir": 0.5404, "F1_UP_FORT": 0.2642, "F1_DOWN_FORT": 0.4065, "train_start": "2000-11-10", "sampler": "SMOTETomek", "best_params": "{}", "features": ["NFCI_ret_5d__div__IWM_SmallCap_vol_20d", "SPY_zscore_60d", "heston_xi__prod__vix_mean_abs_ret_5d", "SJM_JM_Smucker_ret_1d", "vix_momentum_3d__zrel__VZ_ret_5d", "SBUX_zscore_60d", "IWM_SmallCap_vol_20d", "SLB_Schlumberger_ret_5d", "STLFSI4_zscore_60d__ret5x__PPL_PPL_ret_5d", "EBAY_eBay_ret_1d__div__vix_mean_abs_ret_5d", "heston_xi__minus__EBAY_eBay_ret_1d", "NFCI_ret_5d__div__vix_mean_abs_ret_5d", "EXC_Exelon_ret_1d", "EBAY_eBay_ret_1d__ret5x__PPL_PPL_ret_5d"], "is_new": false}, {"model_id": "v1_h1_GLOBAL_GradientBoosting_N15", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 1, "n_features": 15, "F1_dir": 0.5395, "F1_UP_FORT": 0.254, "F1_DOWN_FORT": 0.4151, "train_start": "2000-11-10", "sampler": "SMOTETomek", "best_params": "{}", "features": ["NFCI_ret_5d__div__IWM_SmallCap_vol_20d", "SPY_zscore_60d", "heston_xi__prod__vix_mean_abs_ret_5d", "SJM_JM_Smucker_ret_1d", "vix_momentum_3d__zrel__VZ_ret_5d", "SBUX_zscore_60d", "IWM_SmallCap_vol_20d", "SLB_Schlumberger_ret_5d", "STLFSI4_zscore_60d__ret5x__PPL_PPL_ret_5d", "EBAY_eBay_ret_1d__div__vix_mean_abs_ret_5d", "heston_xi__minus__EBAY_eBay_ret_1d", "NFCI_ret_5d__div__vix_mean_abs_ret_5d", "EXC_Exelon_ret_1d", "EBAY_eBay_ret_1d__ret5x__PPL_PPL_ret_5d", "NFCI_ret_5d__ret5x__vix_momentum_3d"], "is_new": false}, {"model_id": "v1_h1_GLOBAL_GradientBoosting_N16", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 1, "n_features": 16, "F1_dir": 0.5396, "F1_UP_FORT": 0.2678, "F1_DOWN_FORT": 0.4154, "train_start": "2000-11-10", "sampler": "SMOTETomek", "best_params": "{}", "features": ["NFCI_ret_5d__div__IWM_SmallCap_vol_20d", "SPY_zscore_60d", "heston_xi__prod__vix_mean_abs_ret_5d", "SJM_JM_Smucker_ret_1d", "vix_momentum_3d__zrel__VZ_ret_5d", "SBUX_zscore_60d", "IWM_SmallCap_vol_20d", "SLB_Schlumberger_ret_5d", "STLFSI4_zscore_60d__ret5x__PPL_PPL_ret_5d", "EBAY_eBay_ret_1d__div__vix_mean_abs_ret_5d", "heston_xi__minus__EBAY_eBay_ret_1d", "NFCI_ret_5d__div__vix_mean_abs_ret_5d", "EXC_Exelon_ret_1d", "EBAY_eBay_ret_1d__ret5x__PPL_PPL_ret_5d", "NFCI_ret_5d__ret5x__vix_momentum_3d", "heston_xi__div__COST_vol_20d"], "is_new": false}, {"model_id": "v1_h1_GLOBAL_GradientBoosting_N17", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 1, "n_features": 17, "F1_dir": 0.5303, "F1_UP_FORT": 0.2482, "F1_DOWN_FORT": 0.4152, "train_start": "2000-11-10", "sampler": "SMOTETomek", "best_params": "{}", "features": ["NFCI_ret_5d__div__IWM_SmallCap_vol_20d", "SPY_zscore_60d", "heston_xi__prod__vix_mean_abs_ret_5d", "SJM_JM_Smucker_ret_1d", "vix_momentum_3d__zrel__VZ_ret_5d", "SBUX_zscore_60d", "IWM_SmallCap_vol_20d", "SLB_Schlumberger_ret_5d", "STLFSI4_zscore_60d__ret5x__PPL_PPL_ret_5d", "EBAY_eBay_ret_1d__div__vix_mean_abs_ret_5d", "heston_xi__minus__EBAY_eBay_ret_1d", "NFCI_ret_5d__div__vix_mean_abs_ret_5d", "EXC_Exelon_ret_1d", "EBAY_eBay_ret_1d__ret5x__PPL_PPL_ret_5d", "NFCI_ret_5d__ret5x__vix_momentum_3d", "heston_xi__div__COST_vol_20d", "US1Y_Rate_ret_5d"], "is_new": false}, {"model_id": "v1_h1_GLOBAL_GradientBoosting_N18", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 1, "n_features": 18, "F1_dir": 0.5359, "F1_UP_FORT": 0.2437, "F1_DOWN_FORT": 0.422, "train_start": "2000-11-10", "sampler": "SMOTETomek", "best_params": "{}", "features": ["NFCI_ret_5d__div__IWM_SmallCap_vol_20d", "SPY_zscore_60d", "heston_xi__prod__vix_mean_abs_ret_5d", "SJM_JM_Smucker_ret_1d", "vix_momentum_3d__zrel__VZ_ret_5d", "SBUX_zscore_60d", "IWM_SmallCap_vol_20d", "SLB_Schlumberger_ret_5d", "STLFSI4_zscore_60d__ret5x__PPL_PPL_ret_5d", "EBAY_eBay_ret_1d__div__vix_mean_abs_ret_5d", "heston_xi__minus__EBAY_eBay_ret_1d", "NFCI_ret_5d__div__vix_mean_abs_ret_5d", "EXC_Exelon_ret_1d", "EBAY_eBay_ret_1d__ret5x__PPL_PPL_ret_5d", "NFCI_ret_5d__ret5x__vix_momentum_3d", "heston_xi__div__COST_vol_20d", "US1Y_Rate_ret_5d", "IWM_SmallCap_vol_20d__div__BAC_ret_1d"], "is_new": false}, {"model_id": "v1_h1_GLOBAL_GradientBoosting_N19", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 1, "n_features": 19, "F1_dir": 0.5344, "F1_UP_FORT": 0.2526, "F1_DOWN_FORT": 0.4148, "train_start": "2000-11-10", "sampler": "SMOTETomek", "best_params": "{}", "features": ["NFCI_ret_5d__div__IWM_SmallCap_vol_20d", "SPY_zscore_60d", "heston_xi__prod__vix_mean_abs_ret_5d", "SJM_JM_Smucker_ret_1d", "vix_momentum_3d__zrel__VZ_ret_5d", "SBUX_zscore_60d", "IWM_SmallCap_vol_20d", "SLB_Schlumberger_ret_5d", "STLFSI4_zscore_60d__ret5x__PPL_PPL_ret_5d", "EBAY_eBay_ret_1d__div__vix_mean_abs_ret_5d", "heston_xi__minus__EBAY_eBay_ret_1d", "NFCI_ret_5d__div__vix_mean_abs_ret_5d", "EXC_Exelon_ret_1d", "EBAY_eBay_ret_1d__ret5x__PPL_PPL_ret_5d", "NFCI_ret_5d__ret5x__vix_momentum_3d", "heston_xi__div__COST_vol_20d", "US1Y_Rate_ret_5d", "IWM_SmallCap_vol_20d__div__BAC_ret_1d", "ROST_RossStores_ret_1d__minus__COST_vol_20d"], "is_new": false}, {"model_id": "v1_h1_GLOBAL_GradientBoosting_N20", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 1, "n_features": 20, "F1_dir": 0.5416, "F1_UP_FORT": 0.2464, "F1_DOWN_FORT": 0.4101, "train_start": "2000-11-10", "sampler": "SMOTETomek", "best_params": "{}", "features": ["NFCI_ret_5d__div__IWM_SmallCap_vol_20d", "SPY_zscore_60d", "heston_xi__prod__vix_mean_abs_ret_5d", "SJM_JM_Smucker_ret_1d", "vix_momentum_3d__zrel__VZ_ret_5d", "SBUX_zscore_60d", "IWM_SmallCap_vol_20d", "SLB_Schlumberger_ret_5d", "STLFSI4_zscore_60d__ret5x__PPL_PPL_ret_5d", "EBAY_eBay_ret_1d__div__vix_mean_abs_ret_5d", "heston_xi__minus__EBAY_eBay_ret_1d", "NFCI_ret_5d__div__vix_mean_abs_ret_5d", "EXC_Exelon_ret_1d", "EBAY_eBay_ret_1d__ret5x__PPL_PPL_ret_5d", "NFCI_ret_5d__ret5x__vix_momentum_3d", "heston_xi__div__COST_vol_20d", "US1Y_Rate_ret_5d", "IWM_SmallCap_vol_20d__div__BAC_ret_1d", "ROST_RossStores_ret_1d__minus__COST_vol_20d", "STLFSI4_zscore_60d__ret5x__vix_zscore_10d"], "is_new": false}, {"model_id": "v1_h1_GLOBAL_RandomForest_N5", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 1, "n_features": 5, "F1_dir": 0.5559, "F1_UP_FORT": 0.2155, "F1_DOWN_FORT": 0.4295, "train_start": "2000-11-10", "sampler": "SMOTETomek", "best_params": "{}", "features": ["NFCI_ret_5d__div__IWM_SmallCap_vol_20d", "SPY_zscore_60d", "heston_xi__prod__vix_mean_abs_ret_5d", "SJM_JM_Smucker_ret_1d", "vix_momentum_3d__zrel__VZ_ret_5d"], "is_new": false}, {"model_id": "v1_h1_GLOBAL_RandomForest_N6", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 1, "n_features": 6, "F1_dir": 0.5572, "F1_UP_FORT": 0.2324, "F1_DOWN_FORT": 0.4239, "train_start": "2000-11-10", "sampler": "SMOTETomek", "best_params": "{}", "features": ["NFCI_ret_5d__div__IWM_SmallCap_vol_20d", "SPY_zscore_60d", "heston_xi__prod__vix_mean_abs_ret_5d", "SJM_JM_Smucker_ret_1d", "vix_momentum_3d__zrel__VZ_ret_5d", "SBUX_zscore_60d"], "is_new": false}, {"model_id": "v1_h1_GLOBAL_RandomForest_N7", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 1, "n_features": 7, "F1_dir": 0.5562, "F1_UP_FORT": 0.2093, "F1_DOWN_FORT": 0.4011, "train_start": "2000-11-10", "sampler": "SMOTETomek", "best_params": "{}", "features": ["NFCI_ret_5d__div__IWM_SmallCap_vol_20d", "SPY_zscore_60d", "heston_xi__prod__vix_mean_abs_ret_5d", "SJM_JM_Smucker_ret_1d", "vix_momentum_3d__zrel__VZ_ret_5d", "SBUX_zscore_60d", "IWM_SmallCap_vol_20d"], "is_new": false}, {"model_id": "v1_h1_GLOBAL_RandomForest_N8", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 1, "n_features": 8, "F1_dir": 0.5484, "F1_UP_FORT": 0.175, "F1_DOWN_FORT": 0.416, "train_start": "2000-11-10", "sampler": "SMOTETomek", "best_params": "{}", "features": ["NFCI_ret_5d__div__IWM_SmallCap_vol_20d", "SPY_zscore_60d", "heston_xi__prod__vix_mean_abs_ret_5d", "SJM_JM_Smucker_ret_1d", "vix_momentum_3d__zrel__VZ_ret_5d", "SBUX_zscore_60d", "IWM_SmallCap_vol_20d", "SLB_Schlumberger_ret_5d"], "is_new": false}, {"model_id": "v1_h1_GLOBAL_RandomForest_N9", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 1, "n_features": 9, "F1_dir": 0.5471, "F1_UP_FORT": 0.1569, "F1_DOWN_FORT": 0.4139, "train_start": "2000-11-10", "sampler": "SMOTETomek", "best_params": "{}", "features": ["NFCI_ret_5d__div__IWM_SmallCap_vol_20d", "SPY_zscore_60d", "heston_xi__prod__vix_mean_abs_ret_5d", "SJM_JM_Smucker_ret_1d", "vix_momentum_3d__zrel__VZ_ret_5d", "SBUX_zscore_60d", "IWM_SmallCap_vol_20d", "SLB_Schlumberger_ret_5d", "STLFSI4_zscore_60d__ret5x__PPL_PPL_ret_5d"], "is_new": false}, {"model_id": "v1_h1_GLOBAL_RandomForest_N10", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 1, "n_features": 10, "F1_dir": 0.5504, "F1_UP_FORT": 0.167, "F1_DOWN_FORT": 0.4087, "train_start": "2000-11-10", "sampler": "SMOTETomek", "best_params": "{}", "features": ["NFCI_ret_5d__div__IWM_SmallCap_vol_20d", "SPY_zscore_60d", "heston_xi__prod__vix_mean_abs_ret_5d", "SJM_JM_Smucker_ret_1d", "vix_momentum_3d__zrel__VZ_ret_5d", "SBUX_zscore_60d", "IWM_SmallCap_vol_20d", "SLB_Schlumberger_ret_5d", "STLFSI4_zscore_60d__ret5x__PPL_PPL_ret_5d", "EBAY_eBay_ret_1d__div__vix_mean_abs_ret_5d"], "is_new": false}, {"model_id": "v1_h1_GLOBAL_RandomForest_N11", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 1, "n_features": 11, "F1_dir": 0.5548, "F1_UP_FORT": 0.1734, "F1_DOWN_FORT": 0.4383, "train_start": "2000-11-10", "sampler": "SMOTETomek", "best_params": "{}", "features": ["NFCI_ret_5d__div__IWM_SmallCap_vol_20d", "SPY_zscore_60d", "heston_xi__prod__vix_mean_abs_ret_5d", "SJM_JM_Smucker_ret_1d", "vix_momentum_3d__zrel__VZ_ret_5d", "SBUX_zscore_60d", "IWM_SmallCap_vol_20d", "SLB_Schlumberger_ret_5d", "STLFSI4_zscore_60d__ret5x__PPL_PPL_ret_5d", "EBAY_eBay_ret_1d__div__vix_mean_abs_ret_5d", "heston_xi__minus__EBAY_eBay_ret_1d"], "is_new": false}, {"model_id": "v1_h1_GLOBAL_RandomForest_N12", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 1, "n_features": 12, "F1_dir": 0.5605, "F1_UP_FORT": 0.2036, "F1_DOWN_FORT": 0.431, "train_start": "2000-11-10", "sampler": "SMOTETomek", "best_params": "{}", "features": ["NFCI_ret_5d__div__IWM_SmallCap_vol_20d", "SPY_zscore_60d", "heston_xi__prod__vix_mean_abs_ret_5d", "SJM_JM_Smucker_ret_1d", "vix_momentum_3d__zrel__VZ_ret_5d", "SBUX_zscore_60d", "IWM_SmallCap_vol_20d", "SLB_Schlumberger_ret_5d", "STLFSI4_zscore_60d__ret5x__PPL_PPL_ret_5d", "EBAY_eBay_ret_1d__div__vix_mean_abs_ret_5d", "heston_xi__minus__EBAY_eBay_ret_1d", "NFCI_ret_5d__div__vix_mean_abs_ret_5d"], "is_new": false}, {"model_id": "v1_h1_GLOBAL_RandomForest_N13", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 1, "n_features": 13, "F1_dir": 0.5394, "F1_UP_FORT": 0.1988, "F1_DOWN_FORT": 0.434, "train_start": "2000-11-10", "sampler": "SMOTETomek", "best_params": "{}", "features": ["NFCI_ret_5d__div__IWM_SmallCap_vol_20d", "SPY_zscore_60d", "heston_xi__prod__vix_mean_abs_ret_5d", "SJM_JM_Smucker_ret_1d", "vix_momentum_3d__zrel__VZ_ret_5d", "SBUX_zscore_60d", "IWM_SmallCap_vol_20d", "SLB_Schlumberger_ret_5d", "STLFSI4_zscore_60d__ret5x__PPL_PPL_ret_5d", "EBAY_eBay_ret_1d__div__vix_mean_abs_ret_5d", "heston_xi__minus__EBAY_eBay_ret_1d", "NFCI_ret_5d__div__vix_mean_abs_ret_5d", "EXC_Exelon_ret_1d"], "is_new": false}, {"model_id": "v1_h1_GLOBAL_RandomForest_N14", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 1, "n_features": 14, "F1_dir": 0.5399, "F1_UP_FORT": 0.1965, "F1_DOWN_FORT": 0.4179, "train_start": "2000-11-10", "sampler": "SMOTETomek", "best_params": "{}", "features": ["NFCI_ret_5d__div__IWM_SmallCap_vol_20d", "SPY_zscore_60d", "heston_xi__prod__vix_mean_abs_ret_5d", "SJM_JM_Smucker_ret_1d", "vix_momentum_3d__zrel__VZ_ret_5d", "SBUX_zscore_60d", "IWM_SmallCap_vol_20d", "SLB_Schlumberger_ret_5d", "STLFSI4_zscore_60d__ret5x__PPL_PPL_ret_5d", "EBAY_eBay_ret_1d__div__vix_mean_abs_ret_5d", "heston_xi__minus__EBAY_eBay_ret_1d", "NFCI_ret_5d__div__vix_mean_abs_ret_5d", "EXC_Exelon_ret_1d", "EBAY_eBay_ret_1d__ret5x__PPL_PPL_ret_5d"], "is_new": false}, {"model_id": "v1_h1_GLOBAL_RandomForest_N15", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 1, "n_features": 15, "F1_dir": 0.5437, "F1_UP_FORT": 0.2012, "F1_DOWN_FORT": 0.4196, "train_start": "2000-11-10", "sampler": "SMOTETomek", "best_params": "{}", "features": ["NFCI_ret_5d__div__IWM_SmallCap_vol_20d", "SPY_zscore_60d", "heston_xi__prod__vix_mean_abs_ret_5d", "SJM_JM_Smucker_ret_1d", "vix_momentum_3d__zrel__VZ_ret_5d", "SBUX_zscore_60d", "IWM_SmallCap_vol_20d", "SLB_Schlumberger_ret_5d", "STLFSI4_zscore_60d__ret5x__PPL_PPL_ret_5d", "EBAY_eBay_ret_1d__div__vix_mean_abs_ret_5d", "heston_xi__minus__EBAY_eBay_ret_1d", "NFCI_ret_5d__div__vix_mean_abs_ret_5d", "EXC_Exelon_ret_1d", "EBAY_eBay_ret_1d__ret5x__PPL_PPL_ret_5d", "NFCI_ret_5d__ret5x__vix_momentum_3d"], "is_new": false}, {"model_id": "v1_h1_GLOBAL_RandomForest_N16", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 1, "n_features": 16, "F1_dir": 0.551, "F1_UP_FORT": 0.2004, "F1_DOWN_FORT": 0.4452, "train_start": "2000-11-10", "sampler": "SMOTETomek", "best_params": "{}", "features": ["NFCI_ret_5d__div__IWM_SmallCap_vol_20d", "SPY_zscore_60d", "heston_xi__prod__vix_mean_abs_ret_5d", "SJM_JM_Smucker_ret_1d", "vix_momentum_3d__zrel__VZ_ret_5d", "SBUX_zscore_60d", "IWM_SmallCap_vol_20d", "SLB_Schlumberger_ret_5d", "STLFSI4_zscore_60d__ret5x__PPL_PPL_ret_5d", "EBAY_eBay_ret_1d__div__vix_mean_abs_ret_5d", "heston_xi__minus__EBAY_eBay_ret_1d", "NFCI_ret_5d__div__vix_mean_abs_ret_5d", "EXC_Exelon_ret_1d", "EBAY_eBay_ret_1d__ret5x__PPL_PPL_ret_5d", "NFCI_ret_5d__ret5x__vix_momentum_3d", "heston_xi__div__COST_vol_20d"], "is_new": false}, {"model_id": "v1_h1_GLOBAL_RandomForest_N17", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 1, "n_features": 17, "F1_dir": 0.5479, "F1_UP_FORT": 0.2175, "F1_DOWN_FORT": 0.4399, "train_start": "2000-11-10", "sampler": "SMOTETomek", "best_params": "{}", "features": ["NFCI_ret_5d__div__IWM_SmallCap_vol_20d", "SPY_zscore_60d", "heston_xi__prod__vix_mean_abs_ret_5d", "SJM_JM_Smucker_ret_1d", "vix_momentum_3d__zrel__VZ_ret_5d", "SBUX_zscore_60d", "IWM_SmallCap_vol_20d", "SLB_Schlumberger_ret_5d", "STLFSI4_zscore_60d__ret5x__PPL_PPL_ret_5d", "EBAY_eBay_ret_1d__div__vix_mean_abs_ret_5d", "heston_xi__minus__EBAY_eBay_ret_1d", "NFCI_ret_5d__div__vix_mean_abs_ret_5d", "EXC_Exelon_ret_1d", "EBAY_eBay_ret_1d__ret5x__PPL_PPL_ret_5d", "NFCI_ret_5d__ret5x__vix_momentum_3d", "heston_xi__div__COST_vol_20d", "US1Y_Rate_ret_5d"], "is_new": false}, {"model_id": "v1_h1_GLOBAL_RandomForest_N18", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 1, "n_features": 18, "F1_dir": 0.5649, "F1_UP_FORT": 0.2227, "F1_DOWN_FORT": 0.4468, "train_start": "2000-11-10", "sampler": "SMOTETomek", "best_params": "{}", "features": ["NFCI_ret_5d__div__IWM_SmallCap_vol_20d", "SPY_zscore_60d", "heston_xi__prod__vix_mean_abs_ret_5d", "SJM_JM_Smucker_ret_1d", "vix_momentum_3d__zrel__VZ_ret_5d", "SBUX_zscore_60d", "IWM_SmallCap_vol_20d", "SLB_Schlumberger_ret_5d", "STLFSI4_zscore_60d__ret5x__PPL_PPL_ret_5d", "EBAY_eBay_ret_1d__div__vix_mean_abs_ret_5d", "heston_xi__minus__EBAY_eBay_ret_1d", "NFCI_ret_5d__div__vix_mean_abs_ret_5d", "EXC_Exelon_ret_1d", "EBAY_eBay_ret_1d__ret5x__PPL_PPL_ret_5d", "NFCI_ret_5d__ret5x__vix_momentum_3d", "heston_xi__div__COST_vol_20d", "US1Y_Rate_ret_5d", "IWM_SmallCap_vol_20d__div__BAC_ret_1d"], "is_new": false}, {"model_id": "v1_h1_GLOBAL_RandomForest_N19", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 1, "n_features": 19, "F1_dir": 0.5417, "F1_UP_FORT": 0.2055, "F1_DOWN_FORT": 0.4354, "train_start": "2000-11-10", "sampler": "SMOTETomek", "best_params": "{}", "features": ["NFCI_ret_5d__div__IWM_SmallCap_vol_20d", "SPY_zscore_60d", "heston_xi__prod__vix_mean_abs_ret_5d", "SJM_JM_Smucker_ret_1d", "vix_momentum_3d__zrel__VZ_ret_5d", "SBUX_zscore_60d", "IWM_SmallCap_vol_20d", "SLB_Schlumberger_ret_5d", "STLFSI4_zscore_60d__ret5x__PPL_PPL_ret_5d", "EBAY_eBay_ret_1d__div__vix_mean_abs_ret_5d", "heston_xi__minus__EBAY_eBay_ret_1d", "NFCI_ret_5d__div__vix_mean_abs_ret_5d", "EXC_Exelon_ret_1d", "EBAY_eBay_ret_1d__ret5x__PPL_PPL_ret_5d", "NFCI_ret_5d__ret5x__vix_momentum_3d", "heston_xi__div__COST_vol_20d", "US1Y_Rate_ret_5d", "IWM_SmallCap_vol_20d__div__BAC_ret_1d", "ROST_RossStores_ret_1d__minus__COST_vol_20d"], "is_new": false}, {"model_id": "v1_h1_GLOBAL_RandomForest_N20", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 1, "n_features": 20, "F1_dir": 0.5471, "F1_UP_FORT": 0.2074, "F1_DOWN_FORT": 0.4386, "train_start": "2000-11-10", "sampler": "SMOTETomek", "best_params": "{}", "features": ["NFCI_ret_5d__div__IWM_SmallCap_vol_20d", "SPY_zscore_60d", "heston_xi__prod__vix_mean_abs_ret_5d", "SJM_JM_Smucker_ret_1d", "vix_momentum_3d__zrel__VZ_ret_5d", "SBUX_zscore_60d", "IWM_SmallCap_vol_20d", "SLB_Schlumberger_ret_5d", "STLFSI4_zscore_60d__ret5x__PPL_PPL_ret_5d", "EBAY_eBay_ret_1d__div__vix_mean_abs_ret_5d", "heston_xi__minus__EBAY_eBay_ret_1d", "NFCI_ret_5d__div__vix_mean_abs_ret_5d", "EXC_Exelon_ret_1d", "EBAY_eBay_ret_1d__ret5x__PPL_PPL_ret_5d", "NFCI_ret_5d__ret5x__vix_momentum_3d", "heston_xi__div__COST_vol_20d", "US1Y_Rate_ret_5d", "IWM_SmallCap_vol_20d__div__BAC_ret_1d", "ROST_RossStores_ret_1d__minus__COST_vol_20d", "STLFSI4_zscore_60d__ret5x__vix_zscore_10d"], "is_new": false}, {"model_id": "v1_h1_GLOBAL_LogisticRegression_N5", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 1, "n_features": 5, "F1_dir": 0.5316, "F1_UP_FORT": 0.1318, "F1_DOWN_FORT": 0.4138, "train_start": "2000-11-10", "sampler": "SMOTETomek", "best_params": "{}", "features": ["NFCI_ret_5d__div__IWM_SmallCap_vol_20d", "SPY_zscore_60d", "heston_xi__prod__vix_mean_abs_ret_5d", "SJM_JM_Smucker_ret_1d", "vix_momentum_3d__zrel__VZ_ret_5d"], "is_new": false}, {"model_id": "v1_h1_GLOBAL_LogisticRegression_N6", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 1, "n_features": 6, "F1_dir": 0.5372, "F1_UP_FORT": 0.1106, "F1_DOWN_FORT": 0.4187, "train_start": "2000-11-10", "sampler": "SMOTETomek", "best_params": "{}", "features": ["NFCI_ret_5d__div__IWM_SmallCap_vol_20d", "SPY_zscore_60d", "heston_xi__prod__vix_mean_abs_ret_5d", "SJM_JM_Smucker_ret_1d", "vix_momentum_3d__zrel__VZ_ret_5d", "SBUX_zscore_60d"], "is_new": false}, {"model_id": "v1_h1_GLOBAL_LogisticRegression_N7", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 1, "n_features": 7, "F1_dir": 0.5434, "F1_UP_FORT": 0.1224, "F1_DOWN_FORT": 0.4117, "train_start": "2000-11-10", "sampler": "SMOTETomek", "best_params": "{}", "features": ["NFCI_ret_5d__div__IWM_SmallCap_vol_20d", "SPY_zscore_60d", "heston_xi__prod__vix_mean_abs_ret_5d", "SJM_JM_Smucker_ret_1d", "vix_momentum_3d__zrel__VZ_ret_5d", "SBUX_zscore_60d", "IWM_SmallCap_vol_20d"], "is_new": false}, {"model_id": "v1_h1_GLOBAL_LogisticRegression_N8", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 1, "n_features": 8, "F1_dir": 0.5449, "F1_UP_FORT": 0.1303, "F1_DOWN_FORT": 0.4153, "train_start": "2000-11-10", "sampler": "SMOTETomek", "best_params": "{}", "features": ["NFCI_ret_5d__div__IWM_SmallCap_vol_20d", "SPY_zscore_60d", "heston_xi__prod__vix_mean_abs_ret_5d", "SJM_JM_Smucker_ret_1d", "vix_momentum_3d__zrel__VZ_ret_5d", "SBUX_zscore_60d", "IWM_SmallCap_vol_20d", "SLB_Schlumberger_ret_5d"], "is_new": false}, {"model_id": "v1_h1_GLOBAL_LogisticRegression_N9", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 1, "n_features": 9, "F1_dir": 0.5544, "F1_UP_FORT": 0.119, "F1_DOWN_FORT": 0.4174, "train_start": "2000-11-10", "sampler": "SMOTETomek", "best_params": "{}", "features": ["NFCI_ret_5d__div__IWM_SmallCap_vol_20d", "SPY_zscore_60d", "heston_xi__prod__vix_mean_abs_ret_5d", "SJM_JM_Smucker_ret_1d", "vix_momentum_3d__zrel__VZ_ret_5d", "SBUX_zscore_60d", "IWM_SmallCap_vol_20d", "SLB_Schlumberger_ret_5d", "STLFSI4_zscore_60d__ret5x__PPL_PPL_ret_5d"], "is_new": false}, {"model_id": "v1_h1_GLOBAL_LogisticRegression_N10", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 1, "n_features": 10, "F1_dir": 0.5476, "F1_UP_FORT": 0.1445, "F1_DOWN_FORT": 0.4185, "train_start": "2000-11-10", "sampler": "SMOTETomek", "best_params": "{}", "features": ["NFCI_ret_5d__div__IWM_SmallCap_vol_20d", "SPY_zscore_60d", "heston_xi__prod__vix_mean_abs_ret_5d", "SJM_JM_Smucker_ret_1d", "vix_momentum_3d__zrel__VZ_ret_5d", "SBUX_zscore_60d", "IWM_SmallCap_vol_20d", "SLB_Schlumberger_ret_5d", "STLFSI4_zscore_60d__ret5x__PPL_PPL_ret_5d", "EBAY_eBay_ret_1d__div__vix_mean_abs_ret_5d"], "is_new": false}, {"model_id": "v1_h1_GLOBAL_LogisticRegression_N11", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 1, "n_features": 11, "F1_dir": 0.5455, "F1_UP_FORT": 0.2028, "F1_DOWN_FORT": 0.417, "train_start": "2000-11-10", "sampler": "SMOTETomek", "best_params": "{}", "features": ["NFCI_ret_5d__div__IWM_SmallCap_vol_20d", "SPY_zscore_60d", "heston_xi__prod__vix_mean_abs_ret_5d", "SJM_JM_Smucker_ret_1d", "vix_momentum_3d__zrel__VZ_ret_5d", "SBUX_zscore_60d", "IWM_SmallCap_vol_20d", "SLB_Schlumberger_ret_5d", "STLFSI4_zscore_60d__ret5x__PPL_PPL_ret_5d", "EBAY_eBay_ret_1d__div__vix_mean_abs_ret_5d", "heston_xi__minus__EBAY_eBay_ret_1d"], "is_new": false}, {"model_id": "v1_h1_GLOBAL_LogisticRegression_N12", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 1, "n_features": 12, "F1_dir": 0.5438, "F1_UP_FORT": 0.202, "F1_DOWN_FORT": 0.4159, "train_start": "2000-11-10", "sampler": "SMOTETomek", "best_params": "{}", "features": ["NFCI_ret_5d__div__IWM_SmallCap_vol_20d", "SPY_zscore_60d", "heston_xi__prod__vix_mean_abs_ret_5d", "SJM_JM_Smucker_ret_1d", "vix_momentum_3d__zrel__VZ_ret_5d", "SBUX_zscore_60d", "IWM_SmallCap_vol_20d", "SLB_Schlumberger_ret_5d", "STLFSI4_zscore_60d__ret5x__PPL_PPL_ret_5d", "EBAY_eBay_ret_1d__div__vix_mean_abs_ret_5d", "heston_xi__minus__EBAY_eBay_ret_1d", "NFCI_ret_5d__div__vix_mean_abs_ret_5d"], "is_new": false}, {"model_id": "v1_h1_GLOBAL_LogisticRegression_N13", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 1, "n_features": 13, "F1_dir": 0.5443, "F1_UP_FORT": 0.1925, "F1_DOWN_FORT": 0.4059, "train_start": "2000-11-10", "sampler": "SMOTETomek", "best_params": "{}", "features": ["NFCI_ret_5d__div__IWM_SmallCap_vol_20d", "SPY_zscore_60d", "heston_xi__prod__vix_mean_abs_ret_5d", "SJM_JM_Smucker_ret_1d", "vix_momentum_3d__zrel__VZ_ret_5d", "SBUX_zscore_60d", "IWM_SmallCap_vol_20d", "SLB_Schlumberger_ret_5d", "STLFSI4_zscore_60d__ret5x__PPL_PPL_ret_5d", "EBAY_eBay_ret_1d__div__vix_mean_abs_ret_5d", "heston_xi__minus__EBAY_eBay_ret_1d", "NFCI_ret_5d__div__vix_mean_abs_ret_5d", "EXC_Exelon_ret_1d"], "is_new": false}, {"model_id": "v1_h1_GLOBAL_LogisticRegression_N14", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 1, "n_features": 14, "F1_dir": 0.5434, "F1_UP_FORT": 0.1929, "F1_DOWN_FORT": 0.4065, "train_start": "2000-11-10", "sampler": "SMOTETomek", "best_params": "{}", "features": ["NFCI_ret_5d__div__IWM_SmallCap_vol_20d", "SPY_zscore_60d", "heston_xi__prod__vix_mean_abs_ret_5d", "SJM_JM_Smucker_ret_1d", "vix_momentum_3d__zrel__VZ_ret_5d", "SBUX_zscore_60d", "IWM_SmallCap_vol_20d", "SLB_Schlumberger_ret_5d", "STLFSI4_zscore_60d__ret5x__PPL_PPL_ret_5d", "EBAY_eBay_ret_1d__div__vix_mean_abs_ret_5d", "heston_xi__minus__EBAY_eBay_ret_1d", "NFCI_ret_5d__div__vix_mean_abs_ret_5d", "EXC_Exelon_ret_1d", "EBAY_eBay_ret_1d__ret5x__PPL_PPL_ret_5d"], "is_new": false}, {"model_id": "v1_h1_GLOBAL_LogisticRegression_N15", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 1, "n_features": 15, "F1_dir": 0.5493, "F1_UP_FORT": 0.189, "F1_DOWN_FORT": 0.4061, "train_start": "2000-11-10", "sampler": "SMOTETomek", "best_params": "{}", "features": ["NFCI_ret_5d__div__IWM_SmallCap_vol_20d", "SPY_zscore_60d", "heston_xi__prod__vix_mean_abs_ret_5d", "SJM_JM_Smucker_ret_1d", "vix_momentum_3d__zrel__VZ_ret_5d", "SBUX_zscore_60d", "IWM_SmallCap_vol_20d", "SLB_Schlumberger_ret_5d", "STLFSI4_zscore_60d__ret5x__PPL_PPL_ret_5d", "EBAY_eBay_ret_1d__div__vix_mean_abs_ret_5d", "heston_xi__minus__EBAY_eBay_ret_1d", "NFCI_ret_5d__div__vix_mean_abs_ret_5d", "EXC_Exelon_ret_1d", "EBAY_eBay_ret_1d__ret5x__PPL_PPL_ret_5d", "NFCI_ret_5d__ret5x__vix_momentum_3d"], "is_new": false}, {"model_id": "v1_h1_GLOBAL_LogisticRegression_N16", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 1, "n_features": 16, "F1_dir": 0.5411, "F1_UP_FORT": 0.1822, "F1_DOWN_FORT": 0.4167, "train_start": "2000-11-10", "sampler": "SMOTETomek", "best_params": "{}", "features": ["NFCI_ret_5d__div__IWM_SmallCap_vol_20d", "SPY_zscore_60d", "heston_xi__prod__vix_mean_abs_ret_5d", "SJM_JM_Smucker_ret_1d", "vix_momentum_3d__zrel__VZ_ret_5d", "SBUX_zscore_60d", "IWM_SmallCap_vol_20d", "SLB_Schlumberger_ret_5d", "STLFSI4_zscore_60d__ret5x__PPL_PPL_ret_5d", "EBAY_eBay_ret_1d__div__vix_mean_abs_ret_5d", "heston_xi__minus__EBAY_eBay_ret_1d", "NFCI_ret_5d__div__vix_mean_abs_ret_5d", "EXC_Exelon_ret_1d", "EBAY_eBay_ret_1d__ret5x__PPL_PPL_ret_5d", "NFCI_ret_5d__ret5x__vix_momentum_3d", "heston_xi__div__COST_vol_20d"], "is_new": false}, {"model_id": "v1_h1_GLOBAL_LogisticRegression_N17", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 1, "n_features": 17, "F1_dir": 0.5399, "F1_UP_FORT": 0.1737, "F1_DOWN_FORT": 0.4183, "train_start": "2000-11-10", "sampler": "SMOTETomek", "best_params": "{}", "features": ["NFCI_ret_5d__div__IWM_SmallCap_vol_20d", "SPY_zscore_60d", "heston_xi__prod__vix_mean_abs_ret_5d", "SJM_JM_Smucker_ret_1d", "vix_momentum_3d__zrel__VZ_ret_5d", "SBUX_zscore_60d", "IWM_SmallCap_vol_20d", "SLB_Schlumberger_ret_5d", "STLFSI4_zscore_60d__ret5x__PPL_PPL_ret_5d", "EBAY_eBay_ret_1d__div__vix_mean_abs_ret_5d", "heston_xi__minus__EBAY_eBay_ret_1d", "NFCI_ret_5d__div__vix_mean_abs_ret_5d", "EXC_Exelon_ret_1d", "EBAY_eBay_ret_1d__ret5x__PPL_PPL_ret_5d", "NFCI_ret_5d__ret5x__vix_momentum_3d", "heston_xi__div__COST_vol_20d", "US1Y_Rate_ret_5d"], "is_new": false}, {"model_id": "v1_h1_GLOBAL_LogisticRegression_N18", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 1, "n_features": 18, "F1_dir": 0.5515, "F1_UP_FORT": 0.2031, "F1_DOWN_FORT": 0.4087, "train_start": "2000-11-10", "sampler": "SMOTETomek", "best_params": "{}", "features": ["NFCI_ret_5d__div__IWM_SmallCap_vol_20d", "SPY_zscore_60d", "heston_xi__prod__vix_mean_abs_ret_5d", "SJM_JM_Smucker_ret_1d", "vix_momentum_3d__zrel__VZ_ret_5d", "SBUX_zscore_60d", "IWM_SmallCap_vol_20d", "SLB_Schlumberger_ret_5d", "STLFSI4_zscore_60d__ret5x__PPL_PPL_ret_5d", "EBAY_eBay_ret_1d__div__vix_mean_abs_ret_5d", "heston_xi__minus__EBAY_eBay_ret_1d", "NFCI_ret_5d__div__vix_mean_abs_ret_5d", "EXC_Exelon_ret_1d", "EBAY_eBay_ret_1d__ret5x__PPL_PPL_ret_5d", "NFCI_ret_5d__ret5x__vix_momentum_3d", "heston_xi__div__COST_vol_20d", "US1Y_Rate_ret_5d", "IWM_SmallCap_vol_20d__div__BAC_ret_1d"], "is_new": false}, {"model_id": "v1_h1_GLOBAL_LogisticRegression_N19", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 1, "n_features": 19, "F1_dir": 0.5444, "F1_UP_FORT": 0.2066, "F1_DOWN_FORT": 0.4135, "train_start": "2000-11-10", "sampler": "SMOTETomek", "best_params": "{}", "features": ["NFCI_ret_5d__div__IWM_SmallCap_vol_20d", "SPY_zscore_60d", "heston_xi__prod__vix_mean_abs_ret_5d", "SJM_JM_Smucker_ret_1d", "vix_momentum_3d__zrel__VZ_ret_5d", "SBUX_zscore_60d", "IWM_SmallCap_vol_20d", "SLB_Schlumberger_ret_5d", "STLFSI4_zscore_60d__ret5x__PPL_PPL_ret_5d", "EBAY_eBay_ret_1d__div__vix_mean_abs_ret_5d", "heston_xi__minus__EBAY_eBay_ret_1d", "NFCI_ret_5d__div__vix_mean_abs_ret_5d", "EXC_Exelon_ret_1d", "EBAY_eBay_ret_1d__ret5x__PPL_PPL_ret_5d", "NFCI_ret_5d__ret5x__vix_momentum_3d", "heston_xi__div__COST_vol_20d", "US1Y_Rate_ret_5d", "IWM_SmallCap_vol_20d__div__BAC_ret_1d", "ROST_RossStores_ret_1d__minus__COST_vol_20d"], "is_new": false}, {"model_id": "v1_h1_GLOBAL_LogisticRegression_N20", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 1, "n_features": 20, "F1_dir": 0.5457, "F1_UP_FORT": 0.2248, "F1_DOWN_FORT": 0.4141, "train_start": "2000-11-10", "sampler": "SMOTETomek", "best_params": "{}", "features": ["NFCI_ret_5d__div__IWM_SmallCap_vol_20d", "SPY_zscore_60d", "heston_xi__prod__vix_mean_abs_ret_5d", "SJM_JM_Smucker_ret_1d", "vix_momentum_3d__zrel__VZ_ret_5d", "SBUX_zscore_60d", "IWM_SmallCap_vol_20d", "SLB_Schlumberger_ret_5d", "STLFSI4_zscore_60d__ret5x__PPL_PPL_ret_5d", "EBAY_eBay_ret_1d__div__vix_mean_abs_ret_5d", "heston_xi__minus__EBAY_eBay_ret_1d", "NFCI_ret_5d__div__vix_mean_abs_ret_5d", "EXC_Exelon_ret_1d", "EBAY_eBay_ret_1d__ret5x__PPL_PPL_ret_5d", "NFCI_ret_5d__ret5x__vix_momentum_3d", "heston_xi__div__COST_vol_20d", "US1Y_Rate_ret_5d", "IWM_SmallCap_vol_20d__div__BAC_ret_1d", "ROST_RossStores_ret_1d__minus__COST_vol_20d", "STLFSI4_zscore_60d__ret5x__vix_zscore_10d"], "is_new": false}, {"model_id": "v1_h1_GLOBAL_RandomForest_Optuna_N18", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 1, "n_features": 18, "F1_dir": 0.5513, "F1_UP_FORT": 0.2114, "F1_DOWN_FORT": 0.4442, "train_start": "2000-11-10", "sampler": "SMOTETomek", "best_params": "{}", "features": ["NFCI_ret_5d__div__IWM_SmallCap_vol_20d", "SPY_zscore_60d", "heston_xi__prod__vix_mean_abs_ret_5d", "SJM_JM_Smucker_ret_1d", "vix_momentum_3d__zrel__VZ_ret_5d", "SBUX_zscore_60d", "IWM_SmallCap_vol_20d", "SLB_Schlumberger_ret_5d", "STLFSI4_zscore_60d__ret5x__PPL_PPL_ret_5d", "EBAY_eBay_ret_1d__div__vix_mean_abs_ret_5d", "heston_xi__minus__EBAY_eBay_ret_1d", "NFCI_ret_5d__div__vix_mean_abs_ret_5d", "EXC_Exelon_ret_1d", "EBAY_eBay_ret_1d__ret5x__PPL_PPL_ret_5d", "NFCI_ret_5d__ret5x__vix_momentum_3d", "heston_xi__div__COST_vol_20d", "US1Y_Rate_ret_5d", "IWM_SmallCap_vol_20d__div__BAC_ret_1d"], "is_new": false}, {"model_id": "v1_h1_GLOBAL_RandomForest_OptunaCal_N18", "algo": "RandomForestCal", "regime": "GLOBAL", "horizon": 1, "n_features": 18, "F1_dir": 0.5334, "F1_UP_FORT": 0.2413, "F1_DOWN_FORT": 0.4184, "train_start": "2000-11-10", "sampler": "SMOTETomek", "best_params": "{}", "features": ["NFCI_ret_5d__div__IWM_SmallCap_vol_20d", "SPY_zscore_60d", "heston_xi__prod__vix_mean_abs_ret_5d", "SJM_JM_Smucker_ret_1d", "vix_momentum_3d__zrel__VZ_ret_5d", "SBUX_zscore_60d", "IWM_SmallCap_vol_20d", "SLB_Schlumberger_ret_5d", "STLFSI4_zscore_60d__ret5x__PPL_PPL_ret_5d", "EBAY_eBay_ret_1d__div__vix_mean_abs_ret_5d", "heston_xi__minus__EBAY_eBay_ret_1d", "NFCI_ret_5d__div__vix_mean_abs_ret_5d", "EXC_Exelon_ret_1d", "EBAY_eBay_ret_1d__ret5x__PPL_PPL_ret_5d", "NFCI_ret_5d__ret5x__vix_momentum_3d", "heston_xi__div__COST_vol_20d", "US1Y_Rate_ret_5d", "IWM_SmallCap_vol_20d__div__BAC_ret_1d"], "is_new": false}, {"model_id": "v1_h3_CALM_XGBoost_N5", "algo": "XGBoost", "regime": "CALM", "horizon": 3, "n_features": 5, "F1_dir": 0.5143, "F1_UP_FORT": 0.1026, "F1_DOWN_FORT": 0.2617, "train_start": "2000-11-03", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["NFCI_ret_5d", "spx_abs_ret_max_5d", "vix_zscore_10d__div__XLY_Disc_vol_20d", "XLY_Disc_vol_20d", "NFCI_ret_5d__zrel__US3M_Rate_ret_5d"], "is_new": false}, {"model_id": "v1_h3_CALM_XGBoost_N6", "algo": "XGBoost", "regime": "CALM", "horizon": 3, "n_features": 6, "F1_dir": 0.5509, "F1_UP_FORT": 0.1, "F1_DOWN_FORT": 0.2982, "train_start": "2000-11-03", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["NFCI_ret_5d", "spx_abs_ret_max_5d", "vix_zscore_10d__div__XLY_Disc_vol_20d", "XLY_Disc_vol_20d", "NFCI_ret_5d__zrel__US3M_Rate_ret_5d", "NFCI_ret_5d__macross__BLK_BlackRock_ret_1d"], "is_new": false}, {"model_id": "v1_h3_CALM_XGBoost_N7", "algo": "XGBoost", "regime": "CALM", "horizon": 3, "n_features": 7, "F1_dir": 0.5259, "F1_UP_FORT": 0.1667, "F1_DOWN_FORT": 0.2807, "train_start": "2000-11-03", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["NFCI_ret_5d", "spx_abs_ret_max_5d", "vix_zscore_10d__div__XLY_Disc_vol_20d", "XLY_Disc_vol_20d", "NFCI_ret_5d__zrel__US3M_Rate_ret_5d", "NFCI_ret_5d__macross__BLK_BlackRock_ret_1d", "NFCI_ret_5d__minus__LUV_SouthwestAir_vol_20d"], "is_new": false}, {"model_id": "v1_h3_CALM_XGBoost_N8", "algo": "XGBoost", "regime": "CALM", "horizon": 3, "n_features": 8, "F1_dir": 0.5796, "F1_UP_FORT": 0.1649, "F1_DOWN_FORT": 0.3252, "train_start": "2000-11-03", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["NFCI_ret_5d", "spx_abs_ret_max_5d", "vix_zscore_10d__div__XLY_Disc_vol_20d", "XLY_Disc_vol_20d", "NFCI_ret_5d__zrel__US3M_Rate_ret_5d", "NFCI_ret_5d__macross__BLK_BlackRock_ret_1d", "NFCI_ret_5d__minus__LUV_SouthwestAir_vol_20d", "vix_zscore_10d__minus__XLB_Materials_zscore_60d"], "is_new": false}, {"model_id": "v1_h3_CALM_XGBoost_N9", "algo": "XGBoost", "regime": "CALM", "horizon": 3, "n_features": 9, "F1_dir": 0.5628, "F1_UP_FORT": 0.2083, "F1_DOWN_FORT": 0.3802, "train_start": "2000-11-03", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["NFCI_ret_5d", "spx_abs_ret_max_5d", "vix_zscore_10d__div__XLY_Disc_vol_20d", "XLY_Disc_vol_20d", "NFCI_ret_5d__zrel__US3M_Rate_ret_5d", "NFCI_ret_5d__macross__BLK_BlackRock_ret_1d", "NFCI_ret_5d__minus__LUV_SouthwestAir_vol_20d", "vix_zscore_10d__minus__XLB_Materials_zscore_60d", "VIX_Price_ret_20d__div__ROST_RossStores_ret_5d"], "is_new": false}, {"model_id": "v1_h3_CALM_XGBoost_N10", "algo": "XGBoost", "regime": "CALM", "horizon": 3, "n_features": 10, "F1_dir": 0.5464, "F1_UP_FORT": 0.1489, "F1_DOWN_FORT": 0.3564, "train_start": "2000-11-03", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["NFCI_ret_5d", "spx_abs_ret_max_5d", "vix_zscore_10d__div__XLY_Disc_vol_20d", "XLY_Disc_vol_20d", "NFCI_ret_5d__zrel__US3M_Rate_ret_5d", "NFCI_ret_5d__macross__BLK_BlackRock_ret_1d", "NFCI_ret_5d__minus__LUV_SouthwestAir_vol_20d", "vix_zscore_10d__minus__XLB_Materials_zscore_60d", "VIX_Price_ret_20d__div__ROST_RossStores_ret_5d", "NASDAQ_Price_zscore_60d__minus__vix_zscore_10d"], "is_new": false}, {"model_id": "v1_h3_CALM_XGBoost_N11", "algo": "XGBoost", "regime": "CALM", "horizon": 3, "n_features": 11, "F1_dir": 0.5592, "F1_UP_FORT": 0.1443, "F1_DOWN_FORT": 0.3689, "train_start": "2000-11-03", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["NFCI_ret_5d", "spx_abs_ret_max_5d", "vix_zscore_10d__div__XLY_Disc_vol_20d", "XLY_Disc_vol_20d", "NFCI_ret_5d__zrel__US3M_Rate_ret_5d", "NFCI_ret_5d__macross__BLK_BlackRock_ret_1d", "NFCI_ret_5d__minus__LUV_SouthwestAir_vol_20d", "vix_zscore_10d__minus__XLB_Materials_zscore_60d", "VIX_Price_ret_20d__div__ROST_RossStores_ret_5d", "NASDAQ_Price_zscore_60d__minus__vix_zscore_10d", "NASDAQ_Price_zscore_60d__div__XLY_Disc_vol_20d"], "is_new": false}, {"model_id": "v1_h3_CALM_XGBoost_N12", "algo": "XGBoost", "regime": "CALM", "horizon": 3, "n_features": 12, "F1_dir": 0.543, "F1_UP_FORT": 0.1616, "F1_DOWN_FORT": 0.3853, "train_start": "2000-11-03", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["NFCI_ret_5d", "spx_abs_ret_max_5d", "vix_zscore_10d__div__XLY_Disc_vol_20d", "XLY_Disc_vol_20d", "NFCI_ret_5d__zrel__US3M_Rate_ret_5d", "NFCI_ret_5d__macross__BLK_BlackRock_ret_1d", "NFCI_ret_5d__minus__LUV_SouthwestAir_vol_20d", "vix_zscore_10d__minus__XLB_Materials_zscore_60d", "VIX_Price_ret_20d__div__ROST_RossStores_ret_5d", "NASDAQ_Price_zscore_60d__minus__vix_zscore_10d", "NASDAQ_Price_zscore_60d__div__XLY_Disc_vol_20d", "gjr_condvar_h1"], "is_new": false}, {"model_id": "v1_h3_CALM_XGBoost_N13", "algo": "XGBoost", "regime": "CALM", "horizon": 3, "n_features": 13, "F1_dir": 0.5546, "F1_UP_FORT": 0.1702, "F1_DOWN_FORT": 0.32, "train_start": "2000-11-03", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["NFCI_ret_5d", "spx_abs_ret_max_5d", "vix_zscore_10d__div__XLY_Disc_vol_20d", "XLY_Disc_vol_20d", "NFCI_ret_5d__zrel__US3M_Rate_ret_5d", "NFCI_ret_5d__macross__BLK_BlackRock_ret_1d", "NFCI_ret_5d__minus__LUV_SouthwestAir_vol_20d", "vix_zscore_10d__minus__XLB_Materials_zscore_60d", "VIX_Price_ret_20d__div__ROST_RossStores_ret_5d", "NASDAQ_Price_zscore_60d__minus__vix_zscore_10d", "NASDAQ_Price_zscore_60d__div__XLY_Disc_vol_20d", "gjr_condvar_h1", "BTI_BritishAmerican_ret_5d__div__BLK_BlackRock_ret_1d"], "is_new": false}, {"model_id": "v1_h3_CALM_XGBoost_N14", "algo": "XGBoost", "regime": "CALM", "horizon": 3, "n_features": 14, "F1_dir": 0.569, "F1_UP_FORT": 0.1684, "F1_DOWN_FORT": 0.3619, "train_start": "2000-11-03", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["NFCI_ret_5d", "spx_abs_ret_max_5d", "vix_zscore_10d__div__XLY_Disc_vol_20d", "XLY_Disc_vol_20d", "NFCI_ret_5d__zrel__US3M_Rate_ret_5d", "NFCI_ret_5d__macross__BLK_BlackRock_ret_1d", "NFCI_ret_5d__minus__LUV_SouthwestAir_vol_20d", "vix_zscore_10d__minus__XLB_Materials_zscore_60d", "VIX_Price_ret_20d__div__ROST_RossStores_ret_5d", "NASDAQ_Price_zscore_60d__minus__vix_zscore_10d", "NASDAQ_Price_zscore_60d__div__XLY_Disc_vol_20d", "gjr_condvar_h1", "BTI_BritishAmerican_ret_5d__div__BLK_BlackRock_ret_1d", "ASX_Australia_ret_5d"], "is_new": false}, {"model_id": "v1_h3_CALM_XGBoost_N15", "algo": "XGBoost", "regime": "CALM", "horizon": 3, "n_features": 15, "F1_dir": 0.5603, "F1_UP_FORT": 0.2292, "F1_DOWN_FORT": 0.3396, "train_start": "2000-11-03", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["NFCI_ret_5d", "spx_abs_ret_max_5d", "vix_zscore_10d__div__XLY_Disc_vol_20d", "XLY_Disc_vol_20d", "NFCI_ret_5d__zrel__US3M_Rate_ret_5d", "NFCI_ret_5d__macross__BLK_BlackRock_ret_1d", "NFCI_ret_5d__minus__LUV_SouthwestAir_vol_20d", "vix_zscore_10d__minus__XLB_Materials_zscore_60d", "VIX_Price_ret_20d__div__ROST_RossStores_ret_5d", "NASDAQ_Price_zscore_60d__minus__vix_zscore_10d", "NASDAQ_Price_zscore_60d__div__XLY_Disc_vol_20d", "gjr_condvar_h1", "BTI_BritishAmerican_ret_5d__div__BLK_BlackRock_ret_1d", "ASX_Australia_ret_5d", "ROST_RossStores_ret_5d__prod__MKC_McCormick_ret_5d"], "is_new": false}, {"model_id": "v1_h3_CALM_XGBoost_N16", "algo": "XGBoost", "regime": "CALM", "horizon": 3, "n_features": 16, "F1_dir": 0.5682, "F1_UP_FORT": 0.1739, "F1_DOWN_FORT": 0.3455, "train_start": "2000-11-03", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["NFCI_ret_5d", "spx_abs_ret_max_5d", "vix_zscore_10d__div__XLY_Disc_vol_20d", "XLY_Disc_vol_20d", "NFCI_ret_5d__zrel__US3M_Rate_ret_5d", "NFCI_ret_5d__macross__BLK_BlackRock_ret_1d", "NFCI_ret_5d__minus__LUV_SouthwestAir_vol_20d", "vix_zscore_10d__minus__XLB_Materials_zscore_60d", "VIX_Price_ret_20d__div__ROST_RossStores_ret_5d", "NASDAQ_Price_zscore_60d__minus__vix_zscore_10d", "NASDAQ_Price_zscore_60d__div__XLY_Disc_vol_20d", "gjr_condvar_h1", "BTI_BritishAmerican_ret_5d__div__BLK_BlackRock_ret_1d", "ASX_Australia_ret_5d", "ROST_RossStores_ret_5d__prod__MKC_McCormick_ret_5d", "T10Y2Y_Spread_zscore_60d__prod__US3M_Rate_ret_5d"], "is_new": false}, {"model_id": "v1_h3_CALM_XGBoost_N17", "algo": "XGBoost", "regime": "CALM", "horizon": 3, "n_features": 17, "F1_dir": 0.5687, "F1_UP_FORT": 0.1957, "F1_DOWN_FORT": 0.3455, "train_start": "2000-11-03", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["NFCI_ret_5d", "spx_abs_ret_max_5d", "vix_zscore_10d__div__XLY_Disc_vol_20d", "XLY_Disc_vol_20d", "NFCI_ret_5d__zrel__US3M_Rate_ret_5d", "NFCI_ret_5d__macross__BLK_BlackRock_ret_1d", "NFCI_ret_5d__minus__LUV_SouthwestAir_vol_20d", "vix_zscore_10d__minus__XLB_Materials_zscore_60d", "VIX_Price_ret_20d__div__ROST_RossStores_ret_5d", "NASDAQ_Price_zscore_60d__minus__vix_zscore_10d", "NASDAQ_Price_zscore_60d__div__XLY_Disc_vol_20d", "gjr_condvar_h1", "BTI_BritishAmerican_ret_5d__div__BLK_BlackRock_ret_1d", "ASX_Australia_ret_5d", "ROST_RossStores_ret_5d__prod__MKC_McCormick_ret_5d", "T10Y2Y_Spread_zscore_60d__prod__US3M_Rate_ret_5d", "EWH_HongKong_ret_5d"], "is_new": false}, {"model_id": "v1_h3_CALM_XGBoost_N18", "algo": "XGBoost", "regime": "CALM", "horizon": 3, "n_features": 18, "F1_dir": 0.5431, "F1_UP_FORT": 0.2083, "F1_DOWN_FORT": 0.3964, "train_start": "2000-11-03", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["NFCI_ret_5d", "spx_abs_ret_max_5d", "vix_zscore_10d__div__XLY_Disc_vol_20d", "XLY_Disc_vol_20d", "NFCI_ret_5d__zrel__US3M_Rate_ret_5d", "NFCI_ret_5d__macross__BLK_BlackRock_ret_1d", "NFCI_ret_5d__minus__LUV_SouthwestAir_vol_20d", "vix_zscore_10d__minus__XLB_Materials_zscore_60d", "VIX_Price_ret_20d__div__ROST_RossStores_ret_5d", "NASDAQ_Price_zscore_60d__minus__vix_zscore_10d", "NASDAQ_Price_zscore_60d__div__XLY_Disc_vol_20d", "gjr_condvar_h1", "BTI_BritishAmerican_ret_5d__div__BLK_BlackRock_ret_1d", "ASX_Australia_ret_5d", "ROST_RossStores_ret_5d__prod__MKC_McCormick_ret_5d", "T10Y2Y_Spread_zscore_60d__prod__US3M_Rate_ret_5d", "EWH_HongKong_ret_5d", "T10Y2Y_Spread_zscore_60d__ret5x__XLB_Materials_zscore_60d"], "is_new": false}, {"model_id": "v1_h3_CALM_XGBoost_N19", "algo": "XGBoost", "regime": "CALM", "horizon": 3, "n_features": 19, "F1_dir": 0.569, "F1_UP_FORT": 0.1474, "F1_DOWN_FORT": 0.3478, "train_start": "2000-11-03", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["NFCI_ret_5d", "spx_abs_ret_max_5d", "vix_zscore_10d__div__XLY_Disc_vol_20d", "XLY_Disc_vol_20d", "NFCI_ret_5d__zrel__US3M_Rate_ret_5d", "NFCI_ret_5d__macross__BLK_BlackRock_ret_1d", "NFCI_ret_5d__minus__LUV_SouthwestAir_vol_20d", "vix_zscore_10d__minus__XLB_Materials_zscore_60d", "VIX_Price_ret_20d__div__ROST_RossStores_ret_5d", "NASDAQ_Price_zscore_60d__minus__vix_zscore_10d", "NASDAQ_Price_zscore_60d__div__XLY_Disc_vol_20d", "gjr_condvar_h1", "BTI_BritishAmerican_ret_5d__div__BLK_BlackRock_ret_1d", "ASX_Australia_ret_5d", "ROST_RossStores_ret_5d__prod__MKC_McCormick_ret_5d", "T10Y2Y_Spread_zscore_60d__prod__US3M_Rate_ret_5d", "EWH_HongKong_ret_5d", "T10Y2Y_Spread_zscore_60d__ret5x__XLB_Materials_zscore_60d", "NFCI_ret_5d__minus__BLK_BlackRock_ret_1d"], "is_new": false}, {"model_id": "v1_h3_CALM_XGBoost_N20", "algo": "XGBoost", "regime": "CALM", "horizon": 3, "n_features": 20, "F1_dir": 0.5345, "F1_UP_FORT": 0.1333, "F1_DOWN_FORT": 0.3636, "train_start": "2000-11-03", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["NFCI_ret_5d", "spx_abs_ret_max_5d", "vix_zscore_10d__div__XLY_Disc_vol_20d", "XLY_Disc_vol_20d", "NFCI_ret_5d__zrel__US3M_Rate_ret_5d", "NFCI_ret_5d__macross__BLK_BlackRock_ret_1d", "NFCI_ret_5d__minus__LUV_SouthwestAir_vol_20d", "vix_zscore_10d__minus__XLB_Materials_zscore_60d", "VIX_Price_ret_20d__div__ROST_RossStores_ret_5d", "NASDAQ_Price_zscore_60d__minus__vix_zscore_10d", "NASDAQ_Price_zscore_60d__div__XLY_Disc_vol_20d", "gjr_condvar_h1", "BTI_BritishAmerican_ret_5d__div__BLK_BlackRock_ret_1d", "ASX_Australia_ret_5d", "ROST_RossStores_ret_5d__prod__MKC_McCormick_ret_5d", "T10Y2Y_Spread_zscore_60d__prod__US3M_Rate_ret_5d", "EWH_HongKong_ret_5d", "T10Y2Y_Spread_zscore_60d__ret5x__XLB_Materials_zscore_60d", "NFCI_ret_5d__minus__BLK_BlackRock_ret_1d", "ORCL_vol_20d"], "is_new": false}, {"model_id": "v1_h3_CALM_LightGBM_N5", "algo": "LightGBM", "regime": "CALM", "horizon": 3, "n_features": 5, "F1_dir": 0.5109, "F1_UP_FORT": 0.0899, "F1_DOWN_FORT": 0.2957, "train_start": "2000-11-03", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["NFCI_ret_5d", "spx_abs_ret_max_5d", "vix_zscore_10d__div__XLY_Disc_vol_20d", "XLY_Disc_vol_20d", "NFCI_ret_5d__zrel__US3M_Rate_ret_5d"], "is_new": false}, {"model_id": "v1_h3_CALM_LightGBM_N6", "algo": "LightGBM", "regime": "CALM", "horizon": 3, "n_features": 6, "F1_dir": 0.5516, "F1_UP_FORT": 0.1379, "F1_DOWN_FORT": 0.2833, "train_start": "2000-11-03", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["NFCI_ret_5d", "spx_abs_ret_max_5d", "vix_zscore_10d__div__XLY_Disc_vol_20d", "XLY_Disc_vol_20d", "NFCI_ret_5d__zrel__US3M_Rate_ret_5d", "NFCI_ret_5d__macross__BLK_BlackRock_ret_1d"], "is_new": false}, {"model_id": "v1_h3_CALM_LightGBM_N7", "algo": "LightGBM", "regime": "CALM", "horizon": 3, "n_features": 7, "F1_dir": 0.5086, "F1_UP_FORT": 0.1765, "F1_DOWN_FORT": 0.2727, "train_start": "2000-11-03", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["NFCI_ret_5d", "spx_abs_ret_max_5d", "vix_zscore_10d__div__XLY_Disc_vol_20d", "XLY_Disc_vol_20d", "NFCI_ret_5d__zrel__US3M_Rate_ret_5d", "NFCI_ret_5d__macross__BLK_BlackRock_ret_1d", "NFCI_ret_5d__minus__LUV_SouthwestAir_vol_20d"], "is_new": false}, {"model_id": "v1_h3_CALM_LightGBM_N8", "algo": "LightGBM", "regime": "CALM", "horizon": 3, "n_features": 8, "F1_dir": 0.5357, "F1_UP_FORT": 0.14, "F1_DOWN_FORT": 0.2549, "train_start": "2000-11-03", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["NFCI_ret_5d", "spx_abs_ret_max_5d", "vix_zscore_10d__div__XLY_Disc_vol_20d", "XLY_Disc_vol_20d", "NFCI_ret_5d__zrel__US3M_Rate_ret_5d", "NFCI_ret_5d__macross__BLK_BlackRock_ret_1d", "NFCI_ret_5d__minus__LUV_SouthwestAir_vol_20d", "vix_zscore_10d__minus__XLB_Materials_zscore_60d"], "is_new": false}, {"model_id": "v1_h3_CALM_LightGBM_N9", "algo": "LightGBM", "regime": "CALM", "horizon": 3, "n_features": 9, "F1_dir": 0.5501, "F1_UP_FORT": 0.172, "F1_DOWN_FORT": 0.3125, "train_start": "2000-11-03", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["NFCI_ret_5d", "spx_abs_ret_max_5d", "vix_zscore_10d__div__XLY_Disc_vol_20d", "XLY_Disc_vol_20d", "NFCI_ret_5d__zrel__US3M_Rate_ret_5d", "NFCI_ret_5d__macross__BLK_BlackRock_ret_1d", "NFCI_ret_5d__minus__LUV_SouthwestAir_vol_20d", "vix_zscore_10d__minus__XLB_Materials_zscore_60d", "VIX_Price_ret_20d__div__ROST_RossStores_ret_5d"], "is_new": false}, {"model_id": "v1_h3_CALM_LightGBM_N10", "algo": "LightGBM", "regime": "CALM", "horizon": 3, "n_features": 10, "F1_dir": 0.6111, "F1_UP_FORT": 0.233, "F1_DOWN_FORT": 0.3617, "train_start": "2000-11-03", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["NFCI_ret_5d", "spx_abs_ret_max_5d", "vix_zscore_10d__div__XLY_Disc_vol_20d", "XLY_Disc_vol_20d", "NFCI_ret_5d__zrel__US3M_Rate_ret_5d", "NFCI_ret_5d__macross__BLK_BlackRock_ret_1d", "NFCI_ret_5d__minus__LUV_SouthwestAir_vol_20d", "vix_zscore_10d__minus__XLB_Materials_zscore_60d", "VIX_Price_ret_20d__div__ROST_RossStores_ret_5d", "NASDAQ_Price_zscore_60d__minus__vix_zscore_10d"], "is_new": false}, {"model_id": "v1_h3_CALM_LightGBM_N11", "algo": "LightGBM", "regime": "CALM", "horizon": 3, "n_features": 11, "F1_dir": 0.6024, "F1_UP_FORT": 0.2245, "F1_DOWN_FORT": 0.3918, "train_start": "2000-11-03", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["NFCI_ret_5d", "spx_abs_ret_max_5d", "vix_zscore_10d__div__XLY_Disc_vol_20d", "XLY_Disc_vol_20d", "NFCI_ret_5d__zrel__US3M_Rate_ret_5d", "NFCI_ret_5d__macross__BLK_BlackRock_ret_1d", "NFCI_ret_5d__minus__LUV_SouthwestAir_vol_20d", "vix_zscore_10d__minus__XLB_Materials_zscore_60d", "VIX_Price_ret_20d__div__ROST_RossStores_ret_5d", "NASDAQ_Price_zscore_60d__minus__vix_zscore_10d", "NASDAQ_Price_zscore_60d__div__XLY_Disc_vol_20d"], "is_new": false}, {"model_id": "v1_h3_CALM_LightGBM_N12", "algo": "LightGBM", "regime": "CALM", "horizon": 3, "n_features": 12, "F1_dir": 0.5967, "F1_UP_FORT": 0.1905, "F1_DOWN_FORT": 0.3736, "train_start": "2000-11-03", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["NFCI_ret_5d", "spx_abs_ret_max_5d", "vix_zscore_10d__div__XLY_Disc_vol_20d", "XLY_Disc_vol_20d", "NFCI_ret_5d__zrel__US3M_Rate_ret_5d", "NFCI_ret_5d__macross__BLK_BlackRock_ret_1d", "NFCI_ret_5d__minus__LUV_SouthwestAir_vol_20d", "vix_zscore_10d__minus__XLB_Materials_zscore_60d", "VIX_Price_ret_20d__div__ROST_RossStores_ret_5d", "NASDAQ_Price_zscore_60d__minus__vix_zscore_10d", "NASDAQ_Price_zscore_60d__div__XLY_Disc_vol_20d", "gjr_condvar_h1"], "is_new": false}, {"model_id": "v1_h3_CALM_LightGBM_N13", "algo": "LightGBM", "regime": "CALM", "horizon": 3, "n_features": 13, "F1_dir": 0.5801, "F1_UP_FORT": 0.2157, "F1_DOWN_FORT": 0.3125, "train_start": "2000-11-03", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["NFCI_ret_5d", "spx_abs_ret_max_5d", "vix_zscore_10d__div__XLY_Disc_vol_20d", "XLY_Disc_vol_20d", "NFCI_ret_5d__zrel__US3M_Rate_ret_5d", "NFCI_ret_5d__macross__BLK_BlackRock_ret_1d", "NFCI_ret_5d__minus__LUV_SouthwestAir_vol_20d", "vix_zscore_10d__minus__XLB_Materials_zscore_60d", "VIX_Price_ret_20d__div__ROST_RossStores_ret_5d", "NASDAQ_Price_zscore_60d__minus__vix_zscore_10d", "NASDAQ_Price_zscore_60d__div__XLY_Disc_vol_20d", "gjr_condvar_h1", "BTI_BritishAmerican_ret_5d__div__BLK_BlackRock_ret_1d"], "is_new": false}, {"model_id": "v1_h3_CALM_LightGBM_N14", "algo": "LightGBM", "regime": "CALM", "horizon": 3, "n_features": 14, "F1_dir": 0.5387, "F1_UP_FORT": 0.2364, "F1_DOWN_FORT": 0.2979, "train_start": "2000-11-03", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["NFCI_ret_5d", "spx_abs_ret_max_5d", "vix_zscore_10d__div__XLY_Disc_vol_20d", "XLY_Disc_vol_20d", "NFCI_ret_5d__zrel__US3M_Rate_ret_5d", "NFCI_ret_5d__macross__BLK_BlackRock_ret_1d", "NFCI_ret_5d__minus__LUV_SouthwestAir_vol_20d", "vix_zscore_10d__minus__XLB_Materials_zscore_60d", "VIX_Price_ret_20d__div__ROST_RossStores_ret_5d", "NASDAQ_Price_zscore_60d__minus__vix_zscore_10d", "NASDAQ_Price_zscore_60d__div__XLY_Disc_vol_20d", "gjr_condvar_h1", "BTI_BritishAmerican_ret_5d__div__BLK_BlackRock_ret_1d", "ASX_Australia_ret_5d"], "is_new": false}, {"model_id": "v1_h3_CALM_LightGBM_N15", "algo": "LightGBM", "regime": "CALM", "horizon": 3, "n_features": 15, "F1_dir": 0.5733, "F1_UP_FORT": 0.2157, "F1_DOWN_FORT": 0.303, "train_start": "2000-11-03", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["NFCI_ret_5d", "spx_abs_ret_max_5d", "vix_zscore_10d__div__XLY_Disc_vol_20d", "XLY_Disc_vol_20d", "NFCI_ret_5d__zrel__US3M_Rate_ret_5d", "NFCI_ret_5d__macross__BLK_BlackRock_ret_1d", "NFCI_ret_5d__minus__LUV_SouthwestAir_vol_20d", "vix_zscore_10d__minus__XLB_Materials_zscore_60d", "VIX_Price_ret_20d__div__ROST_RossStores_ret_5d", "NASDAQ_Price_zscore_60d__minus__vix_zscore_10d", "NASDAQ_Price_zscore_60d__div__XLY_Disc_vol_20d", "gjr_condvar_h1", "BTI_BritishAmerican_ret_5d__div__BLK_BlackRock_ret_1d", "ASX_Australia_ret_5d", "ROST_RossStores_ret_5d__prod__MKC_McCormick_ret_5d"], "is_new": false}, {"model_id": "v1_h3_CALM_LightGBM_N16", "algo": "LightGBM", "regime": "CALM", "horizon": 3, "n_features": 16, "F1_dir": 0.5455, "F1_UP_FORT": 0.2626, "F1_DOWN_FORT": 0.2766, "train_start": "2000-11-03", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["NFCI_ret_5d", "spx_abs_ret_max_5d", "vix_zscore_10d__div__XLY_Disc_vol_20d", "XLY_Disc_vol_20d", "NFCI_ret_5d__zrel__US3M_Rate_ret_5d", "NFCI_ret_5d__macross__BLK_BlackRock_ret_1d", "NFCI_ret_5d__minus__LUV_SouthwestAir_vol_20d", "vix_zscore_10d__minus__XLB_Materials_zscore_60d", "VIX_Price_ret_20d__div__ROST_RossStores_ret_5d", "NASDAQ_Price_zscore_60d__minus__vix_zscore_10d", "NASDAQ_Price_zscore_60d__div__XLY_Disc_vol_20d", "gjr_condvar_h1", "BTI_BritishAmerican_ret_5d__div__BLK_BlackRock_ret_1d", "ASX_Australia_ret_5d", "ROST_RossStores_ret_5d__prod__MKC_McCormick_ret_5d", "T10Y2Y_Spread_zscore_60d__prod__US3M_Rate_ret_5d"], "is_new": false}, {"model_id": "v1_h3_CALM_LightGBM_N17", "algo": "LightGBM", "regime": "CALM", "horizon": 3, "n_features": 17, "F1_dir": 0.5381, "F1_UP_FORT": 0.1915, "F1_DOWN_FORT": 0.2963, "train_start": "2000-11-03", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["NFCI_ret_5d", "spx_abs_ret_max_5d", "vix_zscore_10d__div__XLY_Disc_vol_20d", "XLY_Disc_vol_20d", "NFCI_ret_5d__zrel__US3M_Rate_ret_5d", "NFCI_ret_5d__macross__BLK_BlackRock_ret_1d", "NFCI_ret_5d__minus__LUV_SouthwestAir_vol_20d", "vix_zscore_10d__minus__XLB_Materials_zscore_60d", "VIX_Price_ret_20d__div__ROST_RossStores_ret_5d", "NASDAQ_Price_zscore_60d__minus__vix_zscore_10d", "NASDAQ_Price_zscore_60d__div__XLY_Disc_vol_20d", "gjr_condvar_h1", "BTI_BritishAmerican_ret_5d__div__BLK_BlackRock_ret_1d", "ASX_Australia_ret_5d", "ROST_RossStores_ret_5d__prod__MKC_McCormick_ret_5d", "T10Y2Y_Spread_zscore_60d__prod__US3M_Rate_ret_5d", "EWH_HongKong_ret_5d"], "is_new": false}, {"model_id": "v1_h3_CALM_LightGBM_N18", "algo": "LightGBM", "regime": "CALM", "horizon": 3, "n_features": 18, "F1_dir": 0.5211, "F1_UP_FORT": 0.1505, "F1_DOWN_FORT": 0.2778, "train_start": "2000-11-03", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["NFCI_ret_5d", "spx_abs_ret_max_5d", "vix_zscore_10d__div__XLY_Disc_vol_20d", "XLY_Disc_vol_20d", "NFCI_ret_5d__zrel__US3M_Rate_ret_5d", "NFCI_ret_5d__macross__BLK_BlackRock_ret_1d", "NFCI_ret_5d__minus__LUV_SouthwestAir_vol_20d", "vix_zscore_10d__minus__XLB_Materials_zscore_60d", "VIX_Price_ret_20d__div__ROST_RossStores_ret_5d", "NASDAQ_Price_zscore_60d__minus__vix_zscore_10d", "NASDAQ_Price_zscore_60d__div__XLY_Disc_vol_20d", "gjr_condvar_h1", "BTI_BritishAmerican_ret_5d__div__BLK_BlackRock_ret_1d", "ASX_Australia_ret_5d", "ROST_RossStores_ret_5d__prod__MKC_McCormick_ret_5d", "T10Y2Y_Spread_zscore_60d__prod__US3M_Rate_ret_5d", "EWH_HongKong_ret_5d", "T10Y2Y_Spread_zscore_60d__ret5x__XLB_Materials_zscore_60d"], "is_new": false}, {"model_id": "v1_h3_CALM_LightGBM_N19", "algo": "LightGBM", "regime": "CALM", "horizon": 3, "n_features": 19, "F1_dir": 0.5163, "F1_UP_FORT": 0.1935, "F1_DOWN_FORT": 0.3, "train_start": "2000-11-03", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["NFCI_ret_5d", "spx_abs_ret_max_5d", "vix_zscore_10d__div__XLY_Disc_vol_20d", "XLY_Disc_vol_20d", "NFCI_ret_5d__zrel__US3M_Rate_ret_5d", "NFCI_ret_5d__macross__BLK_BlackRock_ret_1d", "NFCI_ret_5d__minus__LUV_SouthwestAir_vol_20d", "vix_zscore_10d__minus__XLB_Materials_zscore_60d", "VIX_Price_ret_20d__div__ROST_RossStores_ret_5d", "NASDAQ_Price_zscore_60d__minus__vix_zscore_10d", "NASDAQ_Price_zscore_60d__div__XLY_Disc_vol_20d", "gjr_condvar_h1", "BTI_BritishAmerican_ret_5d__div__BLK_BlackRock_ret_1d", "ASX_Australia_ret_5d", "ROST_RossStores_ret_5d__prod__MKC_McCormick_ret_5d", "T10Y2Y_Spread_zscore_60d__prod__US3M_Rate_ret_5d", "EWH_HongKong_ret_5d", "T10Y2Y_Spread_zscore_60d__ret5x__XLB_Materials_zscore_60d", "NFCI_ret_5d__minus__BLK_BlackRock_ret_1d"], "is_new": false}, {"model_id": "v1_h3_CALM_LightGBM_N20", "algo": "LightGBM", "regime": "CALM", "horizon": 3, "n_features": 20, "F1_dir": 0.5114, "F1_UP_FORT": 0.1667, "F1_DOWN_FORT": 0.2913, "train_start": "2000-11-03", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["NFCI_ret_5d", "spx_abs_ret_max_5d", "vix_zscore_10d__div__XLY_Disc_vol_20d", "XLY_Disc_vol_20d", "NFCI_ret_5d__zrel__US3M_Rate_ret_5d", "NFCI_ret_5d__macross__BLK_BlackRock_ret_1d", "NFCI_ret_5d__minus__LUV_SouthwestAir_vol_20d", "vix_zscore_10d__minus__XLB_Materials_zscore_60d", "VIX_Price_ret_20d__div__ROST_RossStores_ret_5d", "NASDAQ_Price_zscore_60d__minus__vix_zscore_10d", "NASDAQ_Price_zscore_60d__div__XLY_Disc_vol_20d", "gjr_condvar_h1", "BTI_BritishAmerican_ret_5d__div__BLK_BlackRock_ret_1d", "ASX_Australia_ret_5d", "ROST_RossStores_ret_5d__prod__MKC_McCormick_ret_5d", "T10Y2Y_Spread_zscore_60d__prod__US3M_Rate_ret_5d", "EWH_HongKong_ret_5d", "T10Y2Y_Spread_zscore_60d__ret5x__XLB_Materials_zscore_60d", "NFCI_ret_5d__minus__BLK_BlackRock_ret_1d", "ORCL_vol_20d"], "is_new": false}, {"model_id": "v1_h3_CALM_GradientBoosting_N5", "algo": "GradientBoosting", "regime": "CALM", "horizon": 3, "n_features": 5, "F1_dir": 0.5325, "F1_UP_FORT": 0.122, "F1_DOWN_FORT": 0.3548, "train_start": "2000-11-03", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["NFCI_ret_5d", "spx_abs_ret_max_5d", "vix_zscore_10d__div__XLY_Disc_vol_20d", "XLY_Disc_vol_20d", "NFCI_ret_5d__zrel__US3M_Rate_ret_5d"], "is_new": false}, {"model_id": "v1_h3_CALM_GradientBoosting_N6", "algo": "GradientBoosting", "regime": "CALM", "horizon": 3, "n_features": 6, "F1_dir": 0.5467, "F1_UP_FORT": 0.125, "F1_DOWN_FORT": 0.2857, "train_start": "2000-11-03", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["NFCI_ret_5d", "spx_abs_ret_max_5d", "vix_zscore_10d__div__XLY_Disc_vol_20d", "XLY_Disc_vol_20d", "NFCI_ret_5d__zrel__US3M_Rate_ret_5d", "NFCI_ret_5d__macross__BLK_BlackRock_ret_1d"], "is_new": false}, {"model_id": "v1_h3_CALM_GradientBoosting_N7", "algo": "GradientBoosting", "regime": "CALM", "horizon": 3, "n_features": 7, "F1_dir": 0.5904, "F1_UP_FORT": 0.1667, "F1_DOWN_FORT": 0.28, "train_start": "2000-11-03", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["NFCI_ret_5d", "spx_abs_ret_max_5d", "vix_zscore_10d__div__XLY_Disc_vol_20d", "XLY_Disc_vol_20d", "NFCI_ret_5d__zrel__US3M_Rate_ret_5d", "NFCI_ret_5d__macross__BLK_BlackRock_ret_1d", "NFCI_ret_5d__minus__LUV_SouthwestAir_vol_20d"], "is_new": false}, {"model_id": "v1_h3_CALM_GradientBoosting_N8", "algo": "GradientBoosting", "regime": "CALM", "horizon": 3, "n_features": 8, "F1_dir": 0.545, "F1_UP_FORT": 0.1875, "F1_DOWN_FORT": 0.2264, "train_start": "2000-11-03", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["NFCI_ret_5d", "spx_abs_ret_max_5d", "vix_zscore_10d__div__XLY_Disc_vol_20d", "XLY_Disc_vol_20d", "NFCI_ret_5d__zrel__US3M_Rate_ret_5d", "NFCI_ret_5d__macross__BLK_BlackRock_ret_1d", "NFCI_ret_5d__minus__LUV_SouthwestAir_vol_20d", "vix_zscore_10d__minus__XLB_Materials_zscore_60d"], "is_new": false}, {"model_id": "v1_h3_CALM_GradientBoosting_N9", "algo": "GradientBoosting", "regime": "CALM", "horizon": 3, "n_features": 9, "F1_dir": 0.5941, "F1_UP_FORT": 0.202, "F1_DOWN_FORT": 0.3636, "train_start": "2000-11-03", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["NFCI_ret_5d", "spx_abs_ret_max_5d", "vix_zscore_10d__div__XLY_Disc_vol_20d", "XLY_Disc_vol_20d", "NFCI_ret_5d__zrel__US3M_Rate_ret_5d", "NFCI_ret_5d__macross__BLK_BlackRock_ret_1d", "NFCI_ret_5d__minus__LUV_SouthwestAir_vol_20d", "vix_zscore_10d__minus__XLB_Materials_zscore_60d", "VIX_Price_ret_20d__div__ROST_RossStores_ret_5d"], "is_new": false}, {"model_id": "v1_h3_CALM_GradientBoosting_N10", "algo": "GradientBoosting", "regime": "CALM", "horizon": 3, "n_features": 10, "F1_dir": 0.587, "F1_UP_FORT": 0.1584, "F1_DOWN_FORT": 0.3299, "train_start": "2000-11-03", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["NFCI_ret_5d", "spx_abs_ret_max_5d", "vix_zscore_10d__div__XLY_Disc_vol_20d", "XLY_Disc_vol_20d", "NFCI_ret_5d__zrel__US3M_Rate_ret_5d", "NFCI_ret_5d__macross__BLK_BlackRock_ret_1d", "NFCI_ret_5d__minus__LUV_SouthwestAir_vol_20d", "vix_zscore_10d__minus__XLB_Materials_zscore_60d", "VIX_Price_ret_20d__div__ROST_RossStores_ret_5d", "NASDAQ_Price_zscore_60d__minus__vix_zscore_10d"], "is_new": false}, {"model_id": "v1_h3_CALM_GradientBoosting_N11", "algo": "GradientBoosting", "regime": "CALM", "horizon": 3, "n_features": 11, "F1_dir": 0.5999, "F1_UP_FORT": 0.1915, "F1_DOWN_FORT": 0.3762, "train_start": "2000-11-03", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["NFCI_ret_5d", "spx_abs_ret_max_5d", "vix_zscore_10d__div__XLY_Disc_vol_20d", "XLY_Disc_vol_20d", "NFCI_ret_5d__zrel__US3M_Rate_ret_5d", "NFCI_ret_5d__macross__BLK_BlackRock_ret_1d", "NFCI_ret_5d__minus__LUV_SouthwestAir_vol_20d", "vix_zscore_10d__minus__XLB_Materials_zscore_60d", "VIX_Price_ret_20d__div__ROST_RossStores_ret_5d", "NASDAQ_Price_zscore_60d__minus__vix_zscore_10d", "NASDAQ_Price_zscore_60d__div__XLY_Disc_vol_20d"], "is_new": false}, {"model_id": "v1_h3_CALM_GradientBoosting_N12", "algo": "GradientBoosting", "regime": "CALM", "horizon": 3, "n_features": 12, "F1_dir": 0.5862, "F1_UP_FORT": 0.2083, "F1_DOWN_FORT": 0.386, "train_start": "2000-11-03", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["NFCI_ret_5d", "spx_abs_ret_max_5d", "vix_zscore_10d__div__XLY_Disc_vol_20d", "XLY_Disc_vol_20d", "NFCI_ret_5d__zrel__US3M_Rate_ret_5d", "NFCI_ret_5d__macross__BLK_BlackRock_ret_1d", "NFCI_ret_5d__minus__LUV_SouthwestAir_vol_20d", "vix_zscore_10d__minus__XLB_Materials_zscore_60d", "VIX_Price_ret_20d__div__ROST_RossStores_ret_5d", "NASDAQ_Price_zscore_60d__minus__vix_zscore_10d", "NASDAQ_Price_zscore_60d__div__XLY_Disc_vol_20d", "gjr_condvar_h1"], "is_new": false}, {"model_id": "v1_h3_CALM_GradientBoosting_N13", "algo": "GradientBoosting", "regime": "CALM", "horizon": 3, "n_features": 13, "F1_dir": 0.581, "F1_UP_FORT": 0.1957, "F1_DOWN_FORT": 0.3019, "train_start": "2000-11-03", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["NFCI_ret_5d", "spx_abs_ret_max_5d", "vix_zscore_10d__div__XLY_Disc_vol_20d", "XLY_Disc_vol_20d", "NFCI_ret_5d__zrel__US3M_Rate_ret_5d", "NFCI_ret_5d__macross__BLK_BlackRock_ret_1d", "NFCI_ret_5d__minus__LUV_SouthwestAir_vol_20d", "vix_zscore_10d__minus__XLB_Materials_zscore_60d", "VIX_Price_ret_20d__div__ROST_RossStores_ret_5d", "NASDAQ_Price_zscore_60d__minus__vix_zscore_10d", "NASDAQ_Price_zscore_60d__div__XLY_Disc_vol_20d", "gjr_condvar_h1", "BTI_BritishAmerican_ret_5d__div__BLK_BlackRock_ret_1d"], "is_new": false}, {"model_id": "v1_h3_CALM_GradientBoosting_N14", "algo": "GradientBoosting", "regime": "CALM", "horizon": 3, "n_features": 14, "F1_dir": 0.5558, "F1_UP_FORT": 0.1429, "F1_DOWN_FORT": 0.2957, "train_start": "2000-11-03", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["NFCI_ret_5d", "spx_abs_ret_max_5d", "vix_zscore_10d__div__XLY_Disc_vol_20d", "XLY_Disc_vol_20d", "NFCI_ret_5d__zrel__US3M_Rate_ret_5d", "NFCI_ret_5d__macross__BLK_BlackRock_ret_1d", "NFCI_ret_5d__minus__LUV_SouthwestAir_vol_20d", "vix_zscore_10d__minus__XLB_Materials_zscore_60d", "VIX_Price_ret_20d__div__ROST_RossStores_ret_5d", "NASDAQ_Price_zscore_60d__minus__vix_zscore_10d", "NASDAQ_Price_zscore_60d__div__XLY_Disc_vol_20d", "gjr_condvar_h1", "BTI_BritishAmerican_ret_5d__div__BLK_BlackRock_ret_1d", "ASX_Australia_ret_5d"], "is_new": false}, {"model_id": "v1_h3_CALM_GradientBoosting_N15", "algo": "GradientBoosting", "regime": "CALM", "horizon": 3, "n_features": 15, "F1_dir": 0.5171, "F1_UP_FORT": 0.2292, "F1_DOWN_FORT": 0.3137, "train_start": "2000-11-03", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["NFCI_ret_5d", "spx_abs_ret_max_5d", "vix_zscore_10d__div__XLY_Disc_vol_20d", "XLY_Disc_vol_20d", "NFCI_ret_5d__zrel__US3M_Rate_ret_5d", "NFCI_ret_5d__macross__BLK_BlackRock_ret_1d", "NFCI_ret_5d__minus__LUV_SouthwestAir_vol_20d", "vix_zscore_10d__minus__XLB_Materials_zscore_60d", "VIX_Price_ret_20d__div__ROST_RossStores_ret_5d", "NASDAQ_Price_zscore_60d__minus__vix_zscore_10d", "NASDAQ_Price_zscore_60d__div__XLY_Disc_vol_20d", "gjr_condvar_h1", "BTI_BritishAmerican_ret_5d__div__BLK_BlackRock_ret_1d", "ASX_Australia_ret_5d", "ROST_RossStores_ret_5d__prod__MKC_McCormick_ret_5d"], "is_new": false}, {"model_id": "v1_h3_CALM_GradientBoosting_N16", "algo": "GradientBoosting", "regime": "CALM", "horizon": 3, "n_features": 16, "F1_dir": 0.5039, "F1_UP_FORT": 0.1978, "F1_DOWN_FORT": 0.2913, "train_start": "2000-11-03", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["NFCI_ret_5d", "spx_abs_ret_max_5d", "vix_zscore_10d__div__XLY_Disc_vol_20d", "XLY_Disc_vol_20d", "NFCI_ret_5d__zrel__US3M_Rate_ret_5d", "NFCI_ret_5d__macross__BLK_BlackRock_ret_1d", "NFCI_ret_5d__minus__LUV_SouthwestAir_vol_20d", "vix_zscore_10d__minus__XLB_Materials_zscore_60d", "VIX_Price_ret_20d__div__ROST_RossStores_ret_5d", "NASDAQ_Price_zscore_60d__minus__vix_zscore_10d", "NASDAQ_Price_zscore_60d__div__XLY_Disc_vol_20d", "gjr_condvar_h1", "BTI_BritishAmerican_ret_5d__div__BLK_BlackRock_ret_1d", "ASX_Australia_ret_5d", "ROST_RossStores_ret_5d__prod__MKC_McCormick_ret_5d", "T10Y2Y_Spread_zscore_60d__prod__US3M_Rate_ret_5d"], "is_new": false}, {"model_id": "v1_h3_CALM_GradientBoosting_N17", "algo": "GradientBoosting", "regime": "CALM", "horizon": 3, "n_features": 17, "F1_dir": 0.5688, "F1_UP_FORT": 0.1609, "F1_DOWN_FORT": 0.3214, "train_start": "2000-11-03", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["NFCI_ret_5d", "spx_abs_ret_max_5d", "vix_zscore_10d__div__XLY_Disc_vol_20d", "XLY_Disc_vol_20d", "NFCI_ret_5d__zrel__US3M_Rate_ret_5d", "NFCI_ret_5d__macross__BLK_BlackRock_ret_1d", "NFCI_ret_5d__minus__LUV_SouthwestAir_vol_20d", "vix_zscore_10d__minus__XLB_Materials_zscore_60d", "VIX_Price_ret_20d__div__ROST_RossStores_ret_5d", "NASDAQ_Price_zscore_60d__minus__vix_zscore_10d", "NASDAQ_Price_zscore_60d__div__XLY_Disc_vol_20d", "gjr_condvar_h1", "BTI_BritishAmerican_ret_5d__div__BLK_BlackRock_ret_1d", "ASX_Australia_ret_5d", "ROST_RossStores_ret_5d__prod__MKC_McCormick_ret_5d", "T10Y2Y_Spread_zscore_60d__prod__US3M_Rate_ret_5d", "EWH_HongKong_ret_5d"], "is_new": false}, {"model_id": "v1_h3_CALM_GradientBoosting_N18", "algo": "GradientBoosting", "regime": "CALM", "horizon": 3, "n_features": 18, "F1_dir": 0.5253, "F1_UP_FORT": 0.1798, "F1_DOWN_FORT": 0.4037, "train_start": "2000-11-03", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["NFCI_ret_5d", "spx_abs_ret_max_5d", "vix_zscore_10d__div__XLY_Disc_vol_20d", "XLY_Disc_vol_20d", "NFCI_ret_5d__zrel__US3M_Rate_ret_5d", "NFCI_ret_5d__macross__BLK_BlackRock_ret_1d", "NFCI_ret_5d__minus__LUV_SouthwestAir_vol_20d", "vix_zscore_10d__minus__XLB_Materials_zscore_60d", "VIX_Price_ret_20d__div__ROST_RossStores_ret_5d", "NASDAQ_Price_zscore_60d__minus__vix_zscore_10d", "NASDAQ_Price_zscore_60d__div__XLY_Disc_vol_20d", "gjr_condvar_h1", "BTI_BritishAmerican_ret_5d__div__BLK_BlackRock_ret_1d", "ASX_Australia_ret_5d", "ROST_RossStores_ret_5d__prod__MKC_McCormick_ret_5d", "T10Y2Y_Spread_zscore_60d__prod__US3M_Rate_ret_5d", "EWH_HongKong_ret_5d", "T10Y2Y_Spread_zscore_60d__ret5x__XLB_Materials_zscore_60d"], "is_new": false}, {"model_id": "v1_h3_CALM_GradientBoosting_N19", "algo": "GradientBoosting", "regime": "CALM", "horizon": 3, "n_features": 19, "F1_dir": 0.5542, "F1_UP_FORT": 0.1758, "F1_DOWN_FORT": 0.3434, "train_start": "2000-11-03", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["NFCI_ret_5d", "spx_abs_ret_max_5d", "vix_zscore_10d__div__XLY_Disc_vol_20d", "XLY_Disc_vol_20d", "NFCI_ret_5d__zrel__US3M_Rate_ret_5d", "NFCI_ret_5d__macross__BLK_BlackRock_ret_1d", "NFCI_ret_5d__minus__LUV_SouthwestAir_vol_20d", "vix_zscore_10d__minus__XLB_Materials_zscore_60d", "VIX_Price_ret_20d__div__ROST_RossStores_ret_5d", "NASDAQ_Price_zscore_60d__minus__vix_zscore_10d", "NASDAQ_Price_zscore_60d__div__XLY_Disc_vol_20d", "gjr_condvar_h1", "BTI_BritishAmerican_ret_5d__div__BLK_BlackRock_ret_1d", "ASX_Australia_ret_5d", "ROST_RossStores_ret_5d__prod__MKC_McCormick_ret_5d", "T10Y2Y_Spread_zscore_60d__prod__US3M_Rate_ret_5d", "EWH_HongKong_ret_5d", "T10Y2Y_Spread_zscore_60d__ret5x__XLB_Materials_zscore_60d", "NFCI_ret_5d__minus__BLK_BlackRock_ret_1d"], "is_new": false}, {"model_id": "v1_h3_CALM_GradientBoosting_N20", "algo": "GradientBoosting", "regime": "CALM", "horizon": 3, "n_features": 20, "F1_dir": 0.5258, "F1_UP_FORT": 0.1573, "F1_DOWN_FORT": 0.3419, "train_start": "2000-11-03", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["NFCI_ret_5d", "spx_abs_ret_max_5d", "vix_zscore_10d__div__XLY_Disc_vol_20d", "XLY_Disc_vol_20d", "NFCI_ret_5d__zrel__US3M_Rate_ret_5d", "NFCI_ret_5d__macross__BLK_BlackRock_ret_1d", "NFCI_ret_5d__minus__LUV_SouthwestAir_vol_20d", "vix_zscore_10d__minus__XLB_Materials_zscore_60d", "VIX_Price_ret_20d__div__ROST_RossStores_ret_5d", "NASDAQ_Price_zscore_60d__minus__vix_zscore_10d", "NASDAQ_Price_zscore_60d__div__XLY_Disc_vol_20d", "gjr_condvar_h1", "BTI_BritishAmerican_ret_5d__div__BLK_BlackRock_ret_1d", "ASX_Australia_ret_5d", "ROST_RossStores_ret_5d__prod__MKC_McCormick_ret_5d", "T10Y2Y_Spread_zscore_60d__prod__US3M_Rate_ret_5d", "EWH_HongKong_ret_5d", "T10Y2Y_Spread_zscore_60d__ret5x__XLB_Materials_zscore_60d", "NFCI_ret_5d__minus__BLK_BlackRock_ret_1d", "ORCL_vol_20d"], "is_new": false}, {"model_id": "v1_h3_CALM_RandomForest_N6", "algo": "RandomForest", "regime": "CALM", "horizon": 3, "n_features": 6, "F1_dir": 0.5113, "F1_UP_FORT": 0.1446, "F1_DOWN_FORT": 0.3688, "train_start": "2000-11-03", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["NFCI_ret_5d", "spx_abs_ret_max_5d", "vix_zscore_10d__div__XLY_Disc_vol_20d", "XLY_Disc_vol_20d", "NFCI_ret_5d__zrel__US3M_Rate_ret_5d", "NFCI_ret_5d__macross__BLK_BlackRock_ret_1d"], "is_new": false}, {"model_id": "v1_h3_CALM_RandomForest_N7", "algo": "RandomForest", "regime": "CALM", "horizon": 3, "n_features": 7, "F1_dir": 0.5178, "F1_UP_FORT": 0.2524, "F1_DOWN_FORT": 0.4211, "train_start": "2000-11-03", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["NFCI_ret_5d", "spx_abs_ret_max_5d", "vix_zscore_10d__div__XLY_Disc_vol_20d", "XLY_Disc_vol_20d", "NFCI_ret_5d__zrel__US3M_Rate_ret_5d", "NFCI_ret_5d__macross__BLK_BlackRock_ret_1d", "NFCI_ret_5d__minus__LUV_SouthwestAir_vol_20d"], "is_new": false}, {"model_id": "v1_h3_CALM_RandomForest_N8", "algo": "RandomForest", "regime": "CALM", "horizon": 3, "n_features": 8, "F1_dir": 0.5237, "F1_UP_FORT": 0.2178, "F1_DOWN_FORT": 0.4118, "train_start": "2000-11-03", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["NFCI_ret_5d", "spx_abs_ret_max_5d", "vix_zscore_10d__div__XLY_Disc_vol_20d", "XLY_Disc_vol_20d", "NFCI_ret_5d__zrel__US3M_Rate_ret_5d", "NFCI_ret_5d__macross__BLK_BlackRock_ret_1d", "NFCI_ret_5d__minus__LUV_SouthwestAir_vol_20d", "vix_zscore_10d__minus__XLB_Materials_zscore_60d"], "is_new": false}, {"model_id": "v1_h3_CALM_RandomForest_N9", "algo": "RandomForest", "regime": "CALM", "horizon": 3, "n_features": 9, "F1_dir": 0.5637, "F1_UP_FORT": 0.2524, "F1_DOWN_FORT": 0.4375, "train_start": "2000-11-03", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["NFCI_ret_5d", "spx_abs_ret_max_5d", "vix_zscore_10d__div__XLY_Disc_vol_20d", "XLY_Disc_vol_20d", "NFCI_ret_5d__zrel__US3M_Rate_ret_5d", "NFCI_ret_5d__macross__BLK_BlackRock_ret_1d", "NFCI_ret_5d__minus__LUV_SouthwestAir_vol_20d", "vix_zscore_10d__minus__XLB_Materials_zscore_60d", "VIX_Price_ret_20d__div__ROST_RossStores_ret_5d"], "is_new": false}, {"model_id": "v1_h3_CALM_RandomForest_N10", "algo": "RandomForest", "regime": "CALM", "horizon": 3, "n_features": 10, "F1_dir": 0.5512, "F1_UP_FORT": 0.25, "F1_DOWN_FORT": 0.3833, "train_start": "2000-11-03", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["NFCI_ret_5d", "spx_abs_ret_max_5d", "vix_zscore_10d__div__XLY_Disc_vol_20d", "XLY_Disc_vol_20d", "NFCI_ret_5d__zrel__US3M_Rate_ret_5d", "NFCI_ret_5d__macross__BLK_BlackRock_ret_1d", "NFCI_ret_5d__minus__LUV_SouthwestAir_vol_20d", "vix_zscore_10d__minus__XLB_Materials_zscore_60d", "VIX_Price_ret_20d__div__ROST_RossStores_ret_5d", "NASDAQ_Price_zscore_60d__minus__vix_zscore_10d"], "is_new": false}, {"model_id": "v1_h3_CALM_RandomForest_N11", "algo": "RandomForest", "regime": "CALM", "horizon": 3, "n_features": 11, "F1_dir": 0.5423, "F1_UP_FORT": 0.2524, "F1_DOWN_FORT": 0.3934, "train_start": "2000-11-03", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["NFCI_ret_5d", "spx_abs_ret_max_5d", "vix_zscore_10d__div__XLY_Disc_vol_20d", "XLY_Disc_vol_20d", "NFCI_ret_5d__zrel__US3M_Rate_ret_5d", "NFCI_ret_5d__macross__BLK_BlackRock_ret_1d", "NFCI_ret_5d__minus__LUV_SouthwestAir_vol_20d", "vix_zscore_10d__minus__XLB_Materials_zscore_60d", "VIX_Price_ret_20d__div__ROST_RossStores_ret_5d", "NASDAQ_Price_zscore_60d__minus__vix_zscore_10d", "NASDAQ_Price_zscore_60d__div__XLY_Disc_vol_20d"], "is_new": false}, {"model_id": "v1_h3_CALM_RandomForest_N12", "algo": "RandomForest", "regime": "CALM", "horizon": 3, "n_features": 12, "F1_dir": 0.5505, "F1_UP_FORT": 0.2524, "F1_DOWN_FORT": 0.4298, "train_start": "2000-11-03", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["NFCI_ret_5d", "spx_abs_ret_max_5d", "vix_zscore_10d__div__XLY_Disc_vol_20d", "XLY_Disc_vol_20d", "NFCI_ret_5d__zrel__US3M_Rate_ret_5d", "NFCI_ret_5d__macross__BLK_BlackRock_ret_1d", "NFCI_ret_5d__minus__LUV_SouthwestAir_vol_20d", "vix_zscore_10d__minus__XLB_Materials_zscore_60d", "VIX_Price_ret_20d__div__ROST_RossStores_ret_5d", "NASDAQ_Price_zscore_60d__minus__vix_zscore_10d", "NASDAQ_Price_zscore_60d__div__XLY_Disc_vol_20d", "gjr_condvar_h1"], "is_new": false}, {"model_id": "v1_h3_CALM_RandomForest_N13", "algo": "RandomForest", "regime": "CALM", "horizon": 3, "n_features": 13, "F1_dir": 0.5723, "F1_UP_FORT": 0.2524, "F1_DOWN_FORT": 0.4516, "train_start": "2000-11-03", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["NFCI_ret_5d", "spx_abs_ret_max_5d", "vix_zscore_10d__div__XLY_Disc_vol_20d", "XLY_Disc_vol_20d", "NFCI_ret_5d__zrel__US3M_Rate_ret_5d", "NFCI_ret_5d__macross__BLK_BlackRock_ret_1d", "NFCI_ret_5d__minus__LUV_SouthwestAir_vol_20d", "vix_zscore_10d__minus__XLB_Materials_zscore_60d", "VIX_Price_ret_20d__div__ROST_RossStores_ret_5d", "NASDAQ_Price_zscore_60d__minus__vix_zscore_10d", "NASDAQ_Price_zscore_60d__div__XLY_Disc_vol_20d", "gjr_condvar_h1", "BTI_BritishAmerican_ret_5d__div__BLK_BlackRock_ret_1d"], "is_new": false}, {"model_id": "v1_h3_CALM_RandomForest_N14", "algo": "RandomForest", "regime": "CALM", "horizon": 3, "n_features": 14, "F1_dir": 0.5637, "F1_UP_FORT": 0.2667, "F1_DOWN_FORT": 0.4545, "train_start": "2000-11-03", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["NFCI_ret_5d", "spx_abs_ret_max_5d", "vix_zscore_10d__div__XLY_Disc_vol_20d", "XLY_Disc_vol_20d", "NFCI_ret_5d__zrel__US3M_Rate_ret_5d", "NFCI_ret_5d__macross__BLK_BlackRock_ret_1d", "NFCI_ret_5d__minus__LUV_SouthwestAir_vol_20d", "vix_zscore_10d__minus__XLB_Materials_zscore_60d", "VIX_Price_ret_20d__div__ROST_RossStores_ret_5d", "NASDAQ_Price_zscore_60d__minus__vix_zscore_10d", "NASDAQ_Price_zscore_60d__div__XLY_Disc_vol_20d", "gjr_condvar_h1", "BTI_BritishAmerican_ret_5d__div__BLK_BlackRock_ret_1d", "ASX_Australia_ret_5d"], "is_new": false}, {"model_id": "v1_h3_CALM_RandomForest_N15", "algo": "RandomForest", "regime": "CALM", "horizon": 3, "n_features": 15, "F1_dir": 0.5633, "F1_UP_FORT": 0.26, "F1_DOWN_FORT": 0.4355, "train_start": "2000-11-03", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["NFCI_ret_5d", "spx_abs_ret_max_5d", "vix_zscore_10d__div__XLY_Disc_vol_20d", "XLY_Disc_vol_20d", "NFCI_ret_5d__zrel__US3M_Rate_ret_5d", "NFCI_ret_5d__macross__BLK_BlackRock_ret_1d", "NFCI_ret_5d__minus__LUV_SouthwestAir_vol_20d", "vix_zscore_10d__minus__XLB_Materials_zscore_60d", "VIX_Price_ret_20d__div__ROST_RossStores_ret_5d", "NASDAQ_Price_zscore_60d__minus__vix_zscore_10d", "NASDAQ_Price_zscore_60d__div__XLY_Disc_vol_20d", "gjr_condvar_h1", "BTI_BritishAmerican_ret_5d__div__BLK_BlackRock_ret_1d", "ASX_Australia_ret_5d", "ROST_RossStores_ret_5d__prod__MKC_McCormick_ret_5d"], "is_new": false}, {"model_id": "v1_h3_CALM_RandomForest_N16", "algo": "RandomForest", "regime": "CALM", "horizon": 3, "n_features": 16, "F1_dir": 0.5595, "F1_UP_FORT": 0.2376, "F1_DOWN_FORT": 0.4355, "train_start": "2000-11-03", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["NFCI_ret_5d", "spx_abs_ret_max_5d", "vix_zscore_10d__div__XLY_Disc_vol_20d", "XLY_Disc_vol_20d", "NFCI_ret_5d__zrel__US3M_Rate_ret_5d", "NFCI_ret_5d__macross__BLK_BlackRock_ret_1d", "NFCI_ret_5d__minus__LUV_SouthwestAir_vol_20d", "vix_zscore_10d__minus__XLB_Materials_zscore_60d", "VIX_Price_ret_20d__div__ROST_RossStores_ret_5d", "NASDAQ_Price_zscore_60d__minus__vix_zscore_10d", "NASDAQ_Price_zscore_60d__div__XLY_Disc_vol_20d", "gjr_condvar_h1", "BTI_BritishAmerican_ret_5d__div__BLK_BlackRock_ret_1d", "ASX_Australia_ret_5d", "ROST_RossStores_ret_5d__prod__MKC_McCormick_ret_5d", "T10Y2Y_Spread_zscore_60d__prod__US3M_Rate_ret_5d"], "is_new": false}, {"model_id": "v1_h3_CALM_RandomForest_N17", "algo": "RandomForest", "regime": "CALM", "horizon": 3, "n_features": 17, "F1_dir": 0.5896, "F1_UP_FORT": 0.2222, "F1_DOWN_FORT": 0.4167, "train_start": "2000-11-03", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["NFCI_ret_5d", "spx_abs_ret_max_5d", "vix_zscore_10d__div__XLY_Disc_vol_20d", "XLY_Disc_vol_20d", "NFCI_ret_5d__zrel__US3M_Rate_ret_5d", "NFCI_ret_5d__macross__BLK_BlackRock_ret_1d", "NFCI_ret_5d__minus__LUV_SouthwestAir_vol_20d", "vix_zscore_10d__minus__XLB_Materials_zscore_60d", "VIX_Price_ret_20d__div__ROST_RossStores_ret_5d", "NASDAQ_Price_zscore_60d__minus__vix_zscore_10d", "NASDAQ_Price_zscore_60d__div__XLY_Disc_vol_20d", "gjr_condvar_h1", "BTI_BritishAmerican_ret_5d__div__BLK_BlackRock_ret_1d", "ASX_Australia_ret_5d", "ROST_RossStores_ret_5d__prod__MKC_McCormick_ret_5d", "T10Y2Y_Spread_zscore_60d__prod__US3M_Rate_ret_5d", "EWH_HongKong_ret_5d"], "is_new": false}, {"model_id": "v1_h3_CALM_RandomForest_N18", "algo": "RandomForest", "regime": "CALM", "horizon": 3, "n_features": 18, "F1_dir": 0.5796, "F1_UP_FORT": 0.24, "F1_DOWN_FORT": 0.4545, "train_start": "2000-11-03", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["NFCI_ret_5d", "spx_abs_ret_max_5d", "vix_zscore_10d__div__XLY_Disc_vol_20d", "XLY_Disc_vol_20d", "NFCI_ret_5d__zrel__US3M_Rate_ret_5d", "NFCI_ret_5d__macross__BLK_BlackRock_ret_1d", "NFCI_ret_5d__minus__LUV_SouthwestAir_vol_20d", "vix_zscore_10d__minus__XLB_Materials_zscore_60d", "VIX_Price_ret_20d__div__ROST_RossStores_ret_5d", "NASDAQ_Price_zscore_60d__minus__vix_zscore_10d", "NASDAQ_Price_zscore_60d__div__XLY_Disc_vol_20d", "gjr_condvar_h1", "BTI_BritishAmerican_ret_5d__div__BLK_BlackRock_ret_1d", "ASX_Australia_ret_5d", "ROST_RossStores_ret_5d__prod__MKC_McCormick_ret_5d", "T10Y2Y_Spread_zscore_60d__prod__US3M_Rate_ret_5d", "EWH_HongKong_ret_5d", "T10Y2Y_Spread_zscore_60d__ret5x__XLB_Materials_zscore_60d"], "is_new": false}, {"model_id": "v1_h3_CALM_RandomForest_N19", "algo": "RandomForest", "regime": "CALM", "horizon": 3, "n_features": 19, "F1_dir": 0.576, "F1_UP_FORT": 0.2424, "F1_DOWN_FORT": 0.4615, "train_start": "2000-11-03", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["NFCI_ret_5d", "spx_abs_ret_max_5d", "vix_zscore_10d__div__XLY_Disc_vol_20d", "XLY_Disc_vol_20d", "NFCI_ret_5d__zrel__US3M_Rate_ret_5d", "NFCI_ret_5d__macross__BLK_BlackRock_ret_1d", "NFCI_ret_5d__minus__LUV_SouthwestAir_vol_20d", "vix_zscore_10d__minus__XLB_Materials_zscore_60d", "VIX_Price_ret_20d__div__ROST_RossStores_ret_5d", "NASDAQ_Price_zscore_60d__minus__vix_zscore_10d", "NASDAQ_Price_zscore_60d__div__XLY_Disc_vol_20d", "gjr_condvar_h1", "BTI_BritishAmerican_ret_5d__div__BLK_BlackRock_ret_1d", "ASX_Australia_ret_5d", "ROST_RossStores_ret_5d__prod__MKC_McCormick_ret_5d", "T10Y2Y_Spread_zscore_60d__prod__US3M_Rate_ret_5d", "EWH_HongKong_ret_5d", "T10Y2Y_Spread_zscore_60d__ret5x__XLB_Materials_zscore_60d", "NFCI_ret_5d__minus__BLK_BlackRock_ret_1d"], "is_new": false}, {"model_id": "v1_h3_CALM_RandomForest_N20", "algo": "RandomForest", "regime": "CALM", "horizon": 3, "n_features": 20, "F1_dir": 0.5669, "F1_UP_FORT": 0.1895, "F1_DOWN_FORT": 0.4275, "train_start": "2000-11-03", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["NFCI_ret_5d", "spx_abs_ret_max_5d", "vix_zscore_10d__div__XLY_Disc_vol_20d", "XLY_Disc_vol_20d", "NFCI_ret_5d__zrel__US3M_Rate_ret_5d", "NFCI_ret_5d__macross__BLK_BlackRock_ret_1d", "NFCI_ret_5d__minus__LUV_SouthwestAir_vol_20d", "vix_zscore_10d__minus__XLB_Materials_zscore_60d", "VIX_Price_ret_20d__div__ROST_RossStores_ret_5d", "NASDAQ_Price_zscore_60d__minus__vix_zscore_10d", "NASDAQ_Price_zscore_60d__div__XLY_Disc_vol_20d", "gjr_condvar_h1", "BTI_BritishAmerican_ret_5d__div__BLK_BlackRock_ret_1d", "ASX_Australia_ret_5d", "ROST_RossStores_ret_5d__prod__MKC_McCormick_ret_5d", "T10Y2Y_Spread_zscore_60d__prod__US3M_Rate_ret_5d", "EWH_HongKong_ret_5d", "T10Y2Y_Spread_zscore_60d__ret5x__XLB_Materials_zscore_60d", "NFCI_ret_5d__minus__BLK_BlackRock_ret_1d", "ORCL_vol_20d"], "is_new": false}, {"model_id": "v1_h3_CALM_LogisticRegression_N8", "algo": "LogisticRegression", "regime": "CALM", "horizon": 3, "n_features": 8, "F1_dir": 0.5104, "F1_UP_FORT": 0.0506, "F1_DOWN_FORT": 0.39, "train_start": "2000-11-03", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["NFCI_ret_5d", "spx_abs_ret_max_5d", "vix_zscore_10d__div__XLY_Disc_vol_20d", "XLY_Disc_vol_20d", "NFCI_ret_5d__zrel__US3M_Rate_ret_5d", "NFCI_ret_5d__macross__BLK_BlackRock_ret_1d", "NFCI_ret_5d__minus__LUV_SouthwestAir_vol_20d", "vix_zscore_10d__minus__XLB_Materials_zscore_60d"], "is_new": false}, {"model_id": "v1_h3_CALM_LogisticRegression_N9", "algo": "LogisticRegression", "regime": "CALM", "horizon": 3, "n_features": 9, "F1_dir": 0.5104, "F1_UP_FORT": 0.0506, "F1_DOWN_FORT": 0.3858, "train_start": "2000-11-03", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["NFCI_ret_5d", "spx_abs_ret_max_5d", "vix_zscore_10d__div__XLY_Disc_vol_20d", "XLY_Disc_vol_20d", "NFCI_ret_5d__zrel__US3M_Rate_ret_5d", "NFCI_ret_5d__macross__BLK_BlackRock_ret_1d", "NFCI_ret_5d__minus__LUV_SouthwestAir_vol_20d", "vix_zscore_10d__minus__XLB_Materials_zscore_60d", "VIX_Price_ret_20d__div__ROST_RossStores_ret_5d"], "is_new": false}, {"model_id": "v1_h3_CALM_LogisticRegression_N10", "algo": "LogisticRegression", "regime": "CALM", "horizon": 3, "n_features": 10, "F1_dir": 0.5101, "F1_UP_FORT": 0.0494, "F1_DOWN_FORT": 0.3723, "train_start": "2000-11-03", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["NFCI_ret_5d", "spx_abs_ret_max_5d", "vix_zscore_10d__div__XLY_Disc_vol_20d", "XLY_Disc_vol_20d", "NFCI_ret_5d__zrel__US3M_Rate_ret_5d", "NFCI_ret_5d__macross__BLK_BlackRock_ret_1d", "NFCI_ret_5d__minus__LUV_SouthwestAir_vol_20d", "vix_zscore_10d__minus__XLB_Materials_zscore_60d", "VIX_Price_ret_20d__div__ROST_RossStores_ret_5d", "NASDAQ_Price_zscore_60d__minus__vix_zscore_10d"], "is_new": false}, {"model_id": "v1_h3_CALM_LogisticRegression_N11", "algo": "LogisticRegression", "regime": "CALM", "horizon": 3, "n_features": 11, "F1_dir": 0.5052, "F1_UP_FORT": 0.0494, "F1_DOWN_FORT": 0.3665, "train_start": "2000-11-03", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["NFCI_ret_5d", "spx_abs_ret_max_5d", "vix_zscore_10d__div__XLY_Disc_vol_20d", "XLY_Disc_vol_20d", "NFCI_ret_5d__zrel__US3M_Rate_ret_5d", "NFCI_ret_5d__macross__BLK_BlackRock_ret_1d", "NFCI_ret_5d__minus__LUV_SouthwestAir_vol_20d", "vix_zscore_10d__minus__XLB_Materials_zscore_60d", "VIX_Price_ret_20d__div__ROST_RossStores_ret_5d", "NASDAQ_Price_zscore_60d__minus__vix_zscore_10d", "NASDAQ_Price_zscore_60d__div__XLY_Disc_vol_20d"], "is_new": false}, {"model_id": "v1_h3_CALM_LogisticRegression_N12", "algo": "LogisticRegression", "regime": "CALM", "horizon": 3, "n_features": 12, "F1_dir": 0.5155, "F1_UP_FORT": 0.0723, "F1_DOWN_FORT": 0.3673, "train_start": "2000-11-03", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["NFCI_ret_5d", "spx_abs_ret_max_5d", "vix_zscore_10d__div__XLY_Disc_vol_20d", "XLY_Disc_vol_20d", "NFCI_ret_5d__zrel__US3M_Rate_ret_5d", "NFCI_ret_5d__macross__BLK_BlackRock_ret_1d", "NFCI_ret_5d__minus__LUV_SouthwestAir_vol_20d", "vix_zscore_10d__minus__XLB_Materials_zscore_60d", "VIX_Price_ret_20d__div__ROST_RossStores_ret_5d", "NASDAQ_Price_zscore_60d__minus__vix_zscore_10d", "NASDAQ_Price_zscore_60d__div__XLY_Disc_vol_20d", "gjr_condvar_h1"], "is_new": false}, {"model_id": "v1_h3_CALM_LogisticRegression_N13", "algo": "LogisticRegression", "regime": "CALM", "horizon": 3, "n_features": 13, "F1_dir": 0.5205, "F1_UP_FORT": 0.0723, "F1_DOWN_FORT": 0.3692, "train_start": "2000-11-03", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["NFCI_ret_5d", "spx_abs_ret_max_5d", "vix_zscore_10d__div__XLY_Disc_vol_20d", "XLY_Disc_vol_20d", "NFCI_ret_5d__zrel__US3M_Rate_ret_5d", "NFCI_ret_5d__macross__BLK_BlackRock_ret_1d", "NFCI_ret_5d__minus__LUV_SouthwestAir_vol_20d", "vix_zscore_10d__minus__XLB_Materials_zscore_60d", "VIX_Price_ret_20d__div__ROST_RossStores_ret_5d", "NASDAQ_Price_zscore_60d__minus__vix_zscore_10d", "NASDAQ_Price_zscore_60d__div__XLY_Disc_vol_20d", "gjr_condvar_h1", "BTI_BritishAmerican_ret_5d__div__BLK_BlackRock_ret_1d"], "is_new": false}, {"model_id": "v1_h3_CALM_LogisticRegression_N14", "algo": "LogisticRegression", "regime": "CALM", "horizon": 3, "n_features": 14, "F1_dir": 0.523, "F1_UP_FORT": 0.0723, "F1_DOWN_FORT": 0.3684, "train_start": "2000-11-03", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["NFCI_ret_5d", "spx_abs_ret_max_5d", "vix_zscore_10d__div__XLY_Disc_vol_20d", "XLY_Disc_vol_20d", "NFCI_ret_5d__zrel__US3M_Rate_ret_5d", "NFCI_ret_5d__macross__BLK_BlackRock_ret_1d", "NFCI_ret_5d__minus__LUV_SouthwestAir_vol_20d", "vix_zscore_10d__minus__XLB_Materials_zscore_60d", "VIX_Price_ret_20d__div__ROST_RossStores_ret_5d", "NASDAQ_Price_zscore_60d__minus__vix_zscore_10d", "NASDAQ_Price_zscore_60d__div__XLY_Disc_vol_20d", "gjr_condvar_h1", "BTI_BritishAmerican_ret_5d__div__BLK_BlackRock_ret_1d", "ASX_Australia_ret_5d"], "is_new": false}, {"model_id": "v1_h3_CALM_LogisticRegression_N15", "algo": "LogisticRegression", "regime": "CALM", "horizon": 3, "n_features": 15, "F1_dir": 0.5363, "F1_UP_FORT": 0.0941, "F1_DOWN_FORT": 0.3757, "train_start": "2000-11-03", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["NFCI_ret_5d", "spx_abs_ret_max_5d", "vix_zscore_10d__div__XLY_Disc_vol_20d", "XLY_Disc_vol_20d", "NFCI_ret_5d__zrel__US3M_Rate_ret_5d", "NFCI_ret_5d__macross__BLK_BlackRock_ret_1d", "NFCI_ret_5d__minus__LUV_SouthwestAir_vol_20d", "vix_zscore_10d__minus__XLB_Materials_zscore_60d", "VIX_Price_ret_20d__div__ROST_RossStores_ret_5d", "NASDAQ_Price_zscore_60d__minus__vix_zscore_10d", "NASDAQ_Price_zscore_60d__div__XLY_Disc_vol_20d", "gjr_condvar_h1", "BTI_BritishAmerican_ret_5d__div__BLK_BlackRock_ret_1d", "ASX_Australia_ret_5d", "ROST_RossStores_ret_5d__prod__MKC_McCormick_ret_5d"], "is_new": false}, {"model_id": "v1_h3_CALM_LogisticRegression_N16", "algo": "LogisticRegression", "regime": "CALM", "horizon": 3, "n_features": 16, "F1_dir": 0.5412, "F1_UP_FORT": 0.1163, "F1_DOWN_FORT": 0.3757, "train_start": "2000-11-03", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["NFCI_ret_5d", "spx_abs_ret_max_5d", "vix_zscore_10d__div__XLY_Disc_vol_20d", "XLY_Disc_vol_20d", "NFCI_ret_5d__zrel__US3M_Rate_ret_5d", "NFCI_ret_5d__macross__BLK_BlackRock_ret_1d", "NFCI_ret_5d__minus__LUV_SouthwestAir_vol_20d", "vix_zscore_10d__minus__XLB_Materials_zscore_60d", "VIX_Price_ret_20d__div__ROST_RossStores_ret_5d", "NASDAQ_Price_zscore_60d__minus__vix_zscore_10d", "NASDAQ_Price_zscore_60d__div__XLY_Disc_vol_20d", "gjr_condvar_h1", "BTI_BritishAmerican_ret_5d__div__BLK_BlackRock_ret_1d", "ASX_Australia_ret_5d", "ROST_RossStores_ret_5d__prod__MKC_McCormick_ret_5d", "T10Y2Y_Spread_zscore_60d__prod__US3M_Rate_ret_5d"], "is_new": false}, {"model_id": "v1_h3_CALM_LogisticRegression_N17", "algo": "LogisticRegression", "regime": "CALM", "horizon": 3, "n_features": 17, "F1_dir": 0.5421, "F1_UP_FORT": 0.1176, "F1_DOWN_FORT": 0.3636, "train_start": "2000-11-03", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["NFCI_ret_5d", "spx_abs_ret_max_5d", "vix_zscore_10d__div__XLY_Disc_vol_20d", "XLY_Disc_vol_20d", "NFCI_ret_5d__zrel__US3M_Rate_ret_5d", "NFCI_ret_5d__macross__BLK_BlackRock_ret_1d", "NFCI_ret_5d__minus__LUV_SouthwestAir_vol_20d", "vix_zscore_10d__minus__XLB_Materials_zscore_60d", "VIX_Price_ret_20d__div__ROST_RossStores_ret_5d", "NASDAQ_Price_zscore_60d__minus__vix_zscore_10d", "NASDAQ_Price_zscore_60d__div__XLY_Disc_vol_20d", "gjr_condvar_h1", "BTI_BritishAmerican_ret_5d__div__BLK_BlackRock_ret_1d", "ASX_Australia_ret_5d", "ROST_RossStores_ret_5d__prod__MKC_McCormick_ret_5d", "T10Y2Y_Spread_zscore_60d__prod__US3M_Rate_ret_5d", "EWH_HongKong_ret_5d"], "is_new": false}, {"model_id": "v1_h3_CALM_LogisticRegression_N18", "algo": "LogisticRegression", "regime": "CALM", "horizon": 3, "n_features": 18, "F1_dir": 0.5276, "F1_UP_FORT": 0.1395, "F1_DOWN_FORT": 0.3605, "train_start": "2000-11-03", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["NFCI_ret_5d", "spx_abs_ret_max_5d", "vix_zscore_10d__div__XLY_Disc_vol_20d", "XLY_Disc_vol_20d", "NFCI_ret_5d__zrel__US3M_Rate_ret_5d", "NFCI_ret_5d__macross__BLK_BlackRock_ret_1d", "NFCI_ret_5d__minus__LUV_SouthwestAir_vol_20d", "vix_zscore_10d__minus__XLB_Materials_zscore_60d", "VIX_Price_ret_20d__div__ROST_RossStores_ret_5d", "NASDAQ_Price_zscore_60d__minus__vix_zscore_10d", "NASDAQ_Price_zscore_60d__div__XLY_Disc_vol_20d", "gjr_condvar_h1", "BTI_BritishAmerican_ret_5d__div__BLK_BlackRock_ret_1d", "ASX_Australia_ret_5d", "ROST_RossStores_ret_5d__prod__MKC_McCormick_ret_5d", "T10Y2Y_Spread_zscore_60d__prod__US3M_Rate_ret_5d", "EWH_HongKong_ret_5d", "T10Y2Y_Spread_zscore_60d__ret5x__XLB_Materials_zscore_60d"], "is_new": false}, {"model_id": "v1_h3_CALM_LogisticRegression_N19", "algo": "LogisticRegression", "regime": "CALM", "horizon": 3, "n_features": 19, "F1_dir": 0.5276, "F1_UP_FORT": 0.1395, "F1_DOWN_FORT": 0.3626, "train_start": "2000-11-03", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["NFCI_ret_5d", "spx_abs_ret_max_5d", "vix_zscore_10d__div__XLY_Disc_vol_20d", "XLY_Disc_vol_20d", "NFCI_ret_5d__zrel__US3M_Rate_ret_5d", "NFCI_ret_5d__macross__BLK_BlackRock_ret_1d", "NFCI_ret_5d__minus__LUV_SouthwestAir_vol_20d", "vix_zscore_10d__minus__XLB_Materials_zscore_60d", "VIX_Price_ret_20d__div__ROST_RossStores_ret_5d", "NASDAQ_Price_zscore_60d__minus__vix_zscore_10d", "NASDAQ_Price_zscore_60d__div__XLY_Disc_vol_20d", "gjr_condvar_h1", "BTI_BritishAmerican_ret_5d__div__BLK_BlackRock_ret_1d", "ASX_Australia_ret_5d", "ROST_RossStores_ret_5d__prod__MKC_McCormick_ret_5d", "T10Y2Y_Spread_zscore_60d__prod__US3M_Rate_ret_5d", "EWH_HongKong_ret_5d", "T10Y2Y_Spread_zscore_60d__ret5x__XLB_Materials_zscore_60d", "NFCI_ret_5d__minus__BLK_BlackRock_ret_1d"], "is_new": false}, {"model_id": "v1_h3_CALM_LogisticRegression_N20", "algo": "LogisticRegression", "regime": "CALM", "horizon": 3, "n_features": 20, "F1_dir": 0.5205, "F1_UP_FORT": 0.0988, "F1_DOWN_FORT": 0.375, "train_start": "2000-11-03", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["NFCI_ret_5d", "spx_abs_ret_max_5d", "vix_zscore_10d__div__XLY_Disc_vol_20d", "XLY_Disc_vol_20d", "NFCI_ret_5d__zrel__US3M_Rate_ret_5d", "NFCI_ret_5d__macross__BLK_BlackRock_ret_1d", "NFCI_ret_5d__minus__LUV_SouthwestAir_vol_20d", "vix_zscore_10d__minus__XLB_Materials_zscore_60d", "VIX_Price_ret_20d__div__ROST_RossStores_ret_5d", "NASDAQ_Price_zscore_60d__minus__vix_zscore_10d", "NASDAQ_Price_zscore_60d__div__XLY_Disc_vol_20d", "gjr_condvar_h1", "BTI_BritishAmerican_ret_5d__div__BLK_BlackRock_ret_1d", "ASX_Australia_ret_5d", "ROST_RossStores_ret_5d__prod__MKC_McCormick_ret_5d", "T10Y2Y_Spread_zscore_60d__prod__US3M_Rate_ret_5d", "EWH_HongKong_ret_5d", "T10Y2Y_Spread_zscore_60d__ret5x__XLB_Materials_zscore_60d", "NFCI_ret_5d__minus__BLK_BlackRock_ret_1d", "ORCL_vol_20d"], "is_new": false}, {"model_id": "v1_h3_CALM_RandomForest_Optuna_N13", "algo": "RandomForest", "regime": "CALM", "horizon": 3, "n_features": 13, "F1_dir": 0.5431, "F1_UP_FORT": 0.25, "F1_DOWN_FORT": 0.4103, "train_start": "2000-11-03", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["NFCI_ret_5d", "spx_abs_ret_max_5d", "vix_zscore_10d__div__XLY_Disc_vol_20d", "XLY_Disc_vol_20d", "NFCI_ret_5d__zrel__US3M_Rate_ret_5d", "NFCI_ret_5d__macross__BLK_BlackRock_ret_1d", "NFCI_ret_5d__minus__LUV_SouthwestAir_vol_20d", "vix_zscore_10d__minus__XLB_Materials_zscore_60d", "VIX_Price_ret_20d__div__ROST_RossStores_ret_5d", "NASDAQ_Price_zscore_60d__minus__vix_zscore_10d", "NASDAQ_Price_zscore_60d__div__XLY_Disc_vol_20d", "gjr_condvar_h1", "BTI_BritishAmerican_ret_5d__div__BLK_BlackRock_ret_1d"], "is_new": false}, {"model_id": "v1_h3_CALM_RandomForest_OptunaCal_N13", "algo": "RandomForestCal", "regime": "CALM", "horizon": 3, "n_features": 13, "F1_dir": 0.5444, "F1_UP_FORT": 0.2642, "F1_DOWN_FORT": 0.4806, "train_start": "2000-11-03", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["NFCI_ret_5d", "spx_abs_ret_max_5d", "vix_zscore_10d__div__XLY_Disc_vol_20d", "XLY_Disc_vol_20d", "NFCI_ret_5d__zrel__US3M_Rate_ret_5d", "NFCI_ret_5d__macross__BLK_BlackRock_ret_1d", "NFCI_ret_5d__minus__LUV_SouthwestAir_vol_20d", "vix_zscore_10d__minus__XLB_Materials_zscore_60d", "VIX_Price_ret_20d__div__ROST_RossStores_ret_5d", "NASDAQ_Price_zscore_60d__minus__vix_zscore_10d", "NASDAQ_Price_zscore_60d__div__XLY_Disc_vol_20d", "gjr_condvar_h1", "BTI_BritishAmerican_ret_5d__div__BLK_BlackRock_ret_1d"], "is_new": false}, {"model_id": "v1_h3_NORMAL_XGBoost_N6", "algo": "XGBoost", "regime": "NORMAL", "horizon": 3, "n_features": 6, "F1_dir": 0.5023, "F1_UP_FORT": 0.2535, "F1_DOWN_FORT": 0.3032, "train_start": "2007-01-05", "sampler": "SMOTE", "best_params": "{}", "features": ["NFCI_ret_5d__zrel__EQR_Equity_ret_1d", "heston_var_ev_h5__macross__SRE_Sempra_ret_5d", "US6M_Rate_ret_20d", "VVIX_ret_1d__div__vix_acceleration_1d", "SRE_Sempra_ret_5d__macross__VIX_Price_ret_5d", "NFCI_ret_20d__ret5x__VIX_Price_ret_5d"], "is_new": false}, {"model_id": "v1_h3_NORMAL_XGBoost_N7", "algo": "XGBoost", "regime": "NORMAL", "horizon": 3, "n_features": 7, "F1_dir": 0.5087, "F1_UP_FORT": 0.314, "F1_DOWN_FORT": 0.2627, "train_start": "2007-01-05", "sampler": "SMOTE", "best_params": "{}", "features": ["NFCI_ret_5d__zrel__EQR_Equity_ret_1d", "heston_var_ev_h5__macross__SRE_Sempra_ret_5d", "US6M_Rate_ret_20d", "VVIX_ret_1d__div__vix_acceleration_1d", "SRE_Sempra_ret_5d__macross__VIX_Price_ret_5d", "NFCI_ret_20d__ret5x__VIX_Price_ret_5d", "heston_var_ev_h5__ret5x__HD_zscore_60d"], "is_new": false}, {"model_id": "v1_h3_NORMAL_XGBoost_N8", "algo": "XGBoost", "regime": "NORMAL", "horizon": 3, "n_features": 8, "F1_dir": 0.5429, "F1_UP_FORT": 0.2948, "F1_DOWN_FORT": 0.2424, "train_start": "2007-01-05", "sampler": "SMOTE", "best_params": "{}", "features": ["NFCI_ret_5d__zrel__EQR_Equity_ret_1d", "heston_var_ev_h5__macross__SRE_Sempra_ret_5d", "US6M_Rate_ret_20d", "VVIX_ret_1d__div__vix_acceleration_1d", "SRE_Sempra_ret_5d__macross__VIX_Price_ret_5d", "NFCI_ret_20d__ret5x__VIX_Price_ret_5d", "heston_var_ev_h5__ret5x__HD_zscore_60d", "NFCI_ret_5d__div__DAX_Germany_vol_20d"], "is_new": false}, {"model_id": "v1_h3_NORMAL_XGBoost_N11", "algo": "XGBoost", "regime": "NORMAL", "horizon": 3, "n_features": 11, "F1_dir": 0.5115, "F1_UP_FORT": 0.2678, "F1_DOWN_FORT": 0.2682, "train_start": "2007-01-05", "sampler": "SMOTE", "best_params": "{}", "features": ["NFCI_ret_5d__zrel__EQR_Equity_ret_1d", "heston_var_ev_h5__macross__SRE_Sempra_ret_5d", "US6M_Rate_ret_20d", "VVIX_ret_1d__div__vix_acceleration_1d", "SRE_Sempra_ret_5d__macross__VIX_Price_ret_5d", "NFCI_ret_20d__ret5x__VIX_Price_ret_5d", "heston_var_ev_h5__ret5x__HD_zscore_60d", "NFCI_ret_5d__div__DAX_Germany_vol_20d", "NFCI_ret_5d__ret5x__NFCI_ret_20d", "NFCI_ret_20d__div__DAX_Germany_vol_20d", "Nikkei_Japan_vol_20d"], "is_new": false}, {"model_id": "v1_h3_NORMAL_XGBoost_N12", "algo": "XGBoost", "regime": "NORMAL", "horizon": 3, "n_features": 12, "F1_dir": 0.5255, "F1_UP_FORT": 0.2611, "F1_DOWN_FORT": 0.2209, "train_start": "2007-01-05", "sampler": "SMOTE", "best_params": "{}", "features": ["NFCI_ret_5d__zrel__EQR_Equity_ret_1d", "heston_var_ev_h5__macross__SRE_Sempra_ret_5d", "US6M_Rate_ret_20d", "VVIX_ret_1d__div__vix_acceleration_1d", "SRE_Sempra_ret_5d__macross__VIX_Price_ret_5d", "NFCI_ret_20d__ret5x__VIX_Price_ret_5d", "heston_var_ev_h5__ret5x__HD_zscore_60d", "NFCI_ret_5d__div__DAX_Germany_vol_20d", "NFCI_ret_5d__ret5x__NFCI_ret_20d", "NFCI_ret_20d__div__DAX_Germany_vol_20d", "Nikkei_Japan_vol_20d", "WTI_Oil_FRED_zscore_60d"], "is_new": false}, {"model_id": "v1_h3_NORMAL_XGBoost_N13", "algo": "XGBoost", "regime": "NORMAL", "horizon": 3, "n_features": 13, "F1_dir": 0.5216, "F1_UP_FORT": 0.2792, "F1_DOWN_FORT": 0.2125, "train_start": "2007-01-05", "sampler": "SMOTE", "best_params": "{}", "features": ["NFCI_ret_5d__zrel__EQR_Equity_ret_1d", "heston_var_ev_h5__macross__SRE_Sempra_ret_5d", "US6M_Rate_ret_20d", "VVIX_ret_1d__div__vix_acceleration_1d", "SRE_Sempra_ret_5d__macross__VIX_Price_ret_5d", "NFCI_ret_20d__ret5x__VIX_Price_ret_5d", "heston_var_ev_h5__ret5x__HD_zscore_60d", "NFCI_ret_5d__div__DAX_Germany_vol_20d", "NFCI_ret_5d__ret5x__NFCI_ret_20d", "NFCI_ret_20d__div__DAX_Germany_vol_20d", "Nikkei_Japan_vol_20d", "WTI_Oil_FRED_zscore_60d", "NFCI_ret_5d__minus__VVIX_ret_1d"], "is_new": false}, {"model_id": "v1_h3_NORMAL_XGBoost_N14", "algo": "XGBoost", "regime": "NORMAL", "horizon": 3, "n_features": 14, "F1_dir": 0.5286, "F1_UP_FORT": 0.2478, "F1_DOWN_FORT": 0.2556, "train_start": "2007-01-05", "sampler": "SMOTE", "best_params": "{}", "features": ["NFCI_ret_5d__zrel__EQR_Equity_ret_1d", "heston_var_ev_h5__macross__SRE_Sempra_ret_5d", "US6M_Rate_ret_20d", "VVIX_ret_1d__div__vix_acceleration_1d", "SRE_Sempra_ret_5d__macross__VIX_Price_ret_5d", "NFCI_ret_20d__ret5x__VIX_Price_ret_5d", "heston_var_ev_h5__ret5x__HD_zscore_60d", "NFCI_ret_5d__div__DAX_Germany_vol_20d", "NFCI_ret_5d__ret5x__NFCI_ret_20d", "NFCI_ret_20d__div__DAX_Germany_vol_20d", "Nikkei_Japan_vol_20d", "WTI_Oil_FRED_zscore_60d", "NFCI_ret_5d__minus__VVIX_ret_1d", "NOC_Northrop_ret_1d__macross__vix_acceleration_1d"], "is_new": false}, {"model_id": "v1_h3_NORMAL_XGBoost_N15", "algo": "XGBoost", "regime": "NORMAL", "horizon": 3, "n_features": 15, "F1_dir": 0.5294, "F1_UP_FORT": 0.2944, "F1_DOWN_FORT": 0.2455, "train_start": "2007-01-05", "sampler": "SMOTE", "best_params": "{}", "features": ["NFCI_ret_5d__zrel__EQR_Equity_ret_1d", "heston_var_ev_h5__macross__SRE_Sempra_ret_5d", "US6M_Rate_ret_20d", "VVIX_ret_1d__div__vix_acceleration_1d", "SRE_Sempra_ret_5d__macross__VIX_Price_ret_5d", "NFCI_ret_20d__ret5x__VIX_Price_ret_5d", "heston_var_ev_h5__ret5x__HD_zscore_60d", "NFCI_ret_5d__div__DAX_Germany_vol_20d", "NFCI_ret_5d__ret5x__NFCI_ret_20d", "NFCI_ret_20d__div__DAX_Germany_vol_20d", "Nikkei_Japan_vol_20d", "WTI_Oil_FRED_zscore_60d", "NFCI_ret_5d__minus__VVIX_ret_1d", "NOC_Northrop_ret_1d__macross__vix_acceleration_1d", "NFCI_ret_5d__ret5x__EQR_Equity_ret_1d"], "is_new": false}, {"model_id": "v1_h3_NORMAL_XGBoost_N16", "algo": "XGBoost", "regime": "NORMAL", "horizon": 3, "n_features": 16, "F1_dir": 0.5324, "F1_UP_FORT": 0.3288, "F1_DOWN_FORT": 0.2682, "train_start": "2007-01-05", "sampler": "SMOTE", "best_params": "{}", "features": ["NFCI_ret_5d__zrel__EQR_Equity_ret_1d", "heston_var_ev_h5__macross__SRE_Sempra_ret_5d", "US6M_Rate_ret_20d", "VVIX_ret_1d__div__vix_acceleration_1d", "SRE_Sempra_ret_5d__macross__VIX_Price_ret_5d", "NFCI_ret_20d__ret5x__VIX_Price_ret_5d", "heston_var_ev_h5__ret5x__HD_zscore_60d", "NFCI_ret_5d__div__DAX_Germany_vol_20d", "NFCI_ret_5d__ret5x__NFCI_ret_20d", "NFCI_ret_20d__div__DAX_Germany_vol_20d", "Nikkei_Japan_vol_20d", "WTI_Oil_FRED_zscore_60d", "NFCI_ret_5d__minus__VVIX_ret_1d", "NOC_Northrop_ret_1d__macross__vix_acceleration_1d", "NFCI_ret_5d__ret5x__EQR_Equity_ret_1d", "DAX_Germany_vol_20d"], "is_new": false}, {"model_id": "v1_h3_NORMAL_XGBoost_N17", "algo": "XGBoost", "regime": "NORMAL", "horizon": 3, "n_features": 17, "F1_dir": 0.5204, "F1_UP_FORT": 0.3226, "F1_DOWN_FORT": 0.2466, "train_start": "2007-01-05", "sampler": "SMOTE", "best_params": "{}", "features": ["NFCI_ret_5d__zrel__EQR_Equity_ret_1d", "heston_var_ev_h5__macross__SRE_Sempra_ret_5d", "US6M_Rate_ret_20d", "VVIX_ret_1d__div__vix_acceleration_1d", "SRE_Sempra_ret_5d__macross__VIX_Price_ret_5d", "NFCI_ret_20d__ret5x__VIX_Price_ret_5d", "heston_var_ev_h5__ret5x__HD_zscore_60d", "NFCI_ret_5d__div__DAX_Germany_vol_20d", "NFCI_ret_5d__ret5x__NFCI_ret_20d", "NFCI_ret_20d__div__DAX_Germany_vol_20d", "Nikkei_Japan_vol_20d", "WTI_Oil_FRED_zscore_60d", "NFCI_ret_5d__minus__VVIX_ret_1d", "NOC_Northrop_ret_1d__macross__vix_acceleration_1d", "NFCI_ret_5d__ret5x__EQR_Equity_ret_1d", "DAX_Germany_vol_20d", "NFCI_ret_5d__zrel__TM_Telephone_ret_5d"], "is_new": false}, {"model_id": "v1_h3_NORMAL_XGBoost_N18", "algo": "XGBoost", "regime": "NORMAL", "horizon": 3, "n_features": 18, "F1_dir": 0.5014, "F1_UP_FORT": 0.2671, "F1_DOWN_FORT": 0.2913, "train_start": "2007-01-05", "sampler": "SMOTE", "best_params": "{}", "features": ["NFCI_ret_5d__zrel__EQR_Equity_ret_1d", "heston_var_ev_h5__macross__SRE_Sempra_ret_5d", "US6M_Rate_ret_20d", "VVIX_ret_1d__div__vix_acceleration_1d", "SRE_Sempra_ret_5d__macross__VIX_Price_ret_5d", "NFCI_ret_20d__ret5x__VIX_Price_ret_5d", "heston_var_ev_h5__ret5x__HD_zscore_60d", "NFCI_ret_5d__div__DAX_Germany_vol_20d", "NFCI_ret_5d__ret5x__NFCI_ret_20d", "NFCI_ret_20d__div__DAX_Germany_vol_20d", "Nikkei_Japan_vol_20d", "WTI_Oil_FRED_zscore_60d", "NFCI_ret_5d__minus__VVIX_ret_1d", "NOC_Northrop_ret_1d__macross__vix_acceleration_1d", "NFCI_ret_5d__ret5x__EQR_Equity_ret_1d", "DAX_Germany_vol_20d", "NFCI_ret_5d__zrel__TM_Telephone_ret_5d", "EFFR_vol_20d"], "is_new": false}, {"model_id": "v1_h3_NORMAL_LightGBM_N6", "algo": "LightGBM", "regime": "NORMAL", "horizon": 3, "n_features": 6, "F1_dir": 0.5173, "F1_UP_FORT": 0.2889, "F1_DOWN_FORT": 0.2789, "train_start": "2007-01-05", "sampler": "SMOTE", "best_params": "{}", "features": ["NFCI_ret_5d__zrel__EQR_Equity_ret_1d", "heston_var_ev_h5__macross__SRE_Sempra_ret_5d", "US6M_Rate_ret_20d", "VVIX_ret_1d__div__vix_acceleration_1d", "SRE_Sempra_ret_5d__macross__VIX_Price_ret_5d", "NFCI_ret_20d__ret5x__VIX_Price_ret_5d"], "is_new": false}, {"model_id": "v1_h3_NORMAL_LightGBM_N7", "algo": "LightGBM", "regime": "NORMAL", "horizon": 3, "n_features": 7, "F1_dir": 0.503, "F1_UP_FORT": 0.2629, "F1_DOWN_FORT": 0.2522, "train_start": "2007-01-05", "sampler": "SMOTE", "best_params": "{}", "features": ["NFCI_ret_5d__zrel__EQR_Equity_ret_1d", "heston_var_ev_h5__macross__SRE_Sempra_ret_5d", "US6M_Rate_ret_20d", "VVIX_ret_1d__div__vix_acceleration_1d", "SRE_Sempra_ret_5d__macross__VIX_Price_ret_5d", "NFCI_ret_20d__ret5x__VIX_Price_ret_5d", "heston_var_ev_h5__ret5x__HD_zscore_60d"], "is_new": false}, {"model_id": "v1_h3_NORMAL_LightGBM_N8", "algo": "LightGBM", "regime": "NORMAL", "horizon": 3, "n_features": 8, "F1_dir": 0.5209, "F1_UP_FORT": 0.295, "F1_DOWN_FORT": 0.3258, "train_start": "2007-01-05", "sampler": "SMOTE", "best_params": "{}", "features": ["NFCI_ret_5d__zrel__EQR_Equity_ret_1d", "heston_var_ev_h5__macross__SRE_Sempra_ret_5d", "US6M_Rate_ret_20d", "VVIX_ret_1d__div__vix_acceleration_1d", "SRE_Sempra_ret_5d__macross__VIX_Price_ret_5d", "NFCI_ret_20d__ret5x__VIX_Price_ret_5d", "heston_var_ev_h5__ret5x__HD_zscore_60d", "NFCI_ret_5d__div__DAX_Germany_vol_20d"], "is_new": false}, {"model_id": "v1_h3_NORMAL_LightGBM_N12", "algo": "LightGBM", "regime": "NORMAL", "horizon": 3, "n_features": 12, "F1_dir": 0.5299, "F1_UP_FORT": 0.2762, "F1_DOWN_FORT": 0.2398, "train_start": "2007-01-05", "sampler": "SMOTE", "best_params": "{}", "features": ["NFCI_ret_5d__zrel__EQR_Equity_ret_1d", "heston_var_ev_h5__macross__SRE_Sempra_ret_5d", "US6M_Rate_ret_20d", "VVIX_ret_1d__div__vix_acceleration_1d", "SRE_Sempra_ret_5d__macross__VIX_Price_ret_5d", "NFCI_ret_20d__ret5x__VIX_Price_ret_5d", "heston_var_ev_h5__ret5x__HD_zscore_60d", "NFCI_ret_5d__div__DAX_Germany_vol_20d", "NFCI_ret_5d__ret5x__NFCI_ret_20d", "NFCI_ret_20d__div__DAX_Germany_vol_20d", "Nikkei_Japan_vol_20d", "WTI_Oil_FRED_zscore_60d"], "is_new": false}, {"model_id": "v1_h3_NORMAL_LightGBM_N13", "algo": "LightGBM", "regime": "NORMAL", "horizon": 3, "n_features": 13, "F1_dir": 0.5, "F1_UP_FORT": 0.2873, "F1_DOWN_FORT": 0.1923, "train_start": "2007-01-05", "sampler": "SMOTE", "best_params": "{}", "features": ["NFCI_ret_5d__zrel__EQR_Equity_ret_1d", "heston_var_ev_h5__macross__SRE_Sempra_ret_5d", "US6M_Rate_ret_20d", "VVIX_ret_1d__div__vix_acceleration_1d", "SRE_Sempra_ret_5d__macross__VIX_Price_ret_5d", "NFCI_ret_20d__ret5x__VIX_Price_ret_5d", "heston_var_ev_h5__ret5x__HD_zscore_60d", "NFCI_ret_5d__div__DAX_Germany_vol_20d", "NFCI_ret_5d__ret5x__NFCI_ret_20d", "NFCI_ret_20d__div__DAX_Germany_vol_20d", "Nikkei_Japan_vol_20d", "WTI_Oil_FRED_zscore_60d", "NFCI_ret_5d__minus__VVIX_ret_1d"], "is_new": false}, {"model_id": "v1_h3_NORMAL_LightGBM_N14", "algo": "LightGBM", "regime": "NORMAL", "horizon": 3, "n_features": 14, "F1_dir": 0.5385, "F1_UP_FORT": 0.2811, "F1_DOWN_FORT": 0.2374, "train_start": "2007-01-05", "sampler": "SMOTE", "best_params": "{}", "features": ["NFCI_ret_5d__zrel__EQR_Equity_ret_1d", "heston_var_ev_h5__macross__SRE_Sempra_ret_5d", "US6M_Rate_ret_20d", "VVIX_ret_1d__div__vix_acceleration_1d", "SRE_Sempra_ret_5d__macross__VIX_Price_ret_5d", "NFCI_ret_20d__ret5x__VIX_Price_ret_5d", "heston_var_ev_h5__ret5x__HD_zscore_60d", "NFCI_ret_5d__div__DAX_Germany_vol_20d", "NFCI_ret_5d__ret5x__NFCI_ret_20d", "NFCI_ret_20d__div__DAX_Germany_vol_20d", "Nikkei_Japan_vol_20d", "WTI_Oil_FRED_zscore_60d", "NFCI_ret_5d__minus__VVIX_ret_1d", "NOC_Northrop_ret_1d__macross__vix_acceleration_1d"], "is_new": false}, {"model_id": "v1_h3_NORMAL_LightGBM_N15", "algo": "LightGBM", "regime": "NORMAL", "horizon": 3, "n_features": 15, "F1_dir": 0.5206, "F1_UP_FORT": 0.2537, "F1_DOWN_FORT": 0.2899, "train_start": "2007-01-05", "sampler": "SMOTE", "best_params": "{}", "features": ["NFCI_ret_5d__zrel__EQR_Equity_ret_1d", "heston_var_ev_h5__macross__SRE_Sempra_ret_5d", "US6M_Rate_ret_20d", "VVIX_ret_1d__div__vix_acceleration_1d", "SRE_Sempra_ret_5d__macross__VIX_Price_ret_5d", "NFCI_ret_20d__ret5x__VIX_Price_ret_5d", "heston_var_ev_h5__ret5x__HD_zscore_60d", "NFCI_ret_5d__div__DAX_Germany_vol_20d", "NFCI_ret_5d__ret5x__NFCI_ret_20d", "NFCI_ret_20d__div__DAX_Germany_vol_20d", "Nikkei_Japan_vol_20d", "WTI_Oil_FRED_zscore_60d", "NFCI_ret_5d__minus__VVIX_ret_1d", "NOC_Northrop_ret_1d__macross__vix_acceleration_1d", "NFCI_ret_5d__ret5x__EQR_Equity_ret_1d"], "is_new": false}, {"model_id": "v1_h3_NORMAL_LightGBM_N16", "algo": "LightGBM", "regime": "NORMAL", "horizon": 3, "n_features": 16, "F1_dir": 0.5201, "F1_UP_FORT": 0.3077, "F1_DOWN_FORT": 0.2347, "train_start": "2007-01-05", "sampler": "SMOTE", "best_params": "{}", "features": ["NFCI_ret_5d__zrel__EQR_Equity_ret_1d", "heston_var_ev_h5__macross__SRE_Sempra_ret_5d", "US6M_Rate_ret_20d", "VVIX_ret_1d__div__vix_acceleration_1d", "SRE_Sempra_ret_5d__macross__VIX_Price_ret_5d", "NFCI_ret_20d__ret5x__VIX_Price_ret_5d", "heston_var_ev_h5__ret5x__HD_zscore_60d", "NFCI_ret_5d__div__DAX_Germany_vol_20d", "NFCI_ret_5d__ret5x__NFCI_ret_20d", "NFCI_ret_20d__div__DAX_Germany_vol_20d", "Nikkei_Japan_vol_20d", "WTI_Oil_FRED_zscore_60d", "NFCI_ret_5d__minus__VVIX_ret_1d", "NOC_Northrop_ret_1d__macross__vix_acceleration_1d", "NFCI_ret_5d__ret5x__EQR_Equity_ret_1d", "DAX_Germany_vol_20d"], "is_new": false}, {"model_id": "v1_h3_NORMAL_LightGBM_N17", "algo": "LightGBM", "regime": "NORMAL", "horizon": 3, "n_features": 17, "F1_dir": 0.5037, "F1_UP_FORT": 0.2849, "F1_DOWN_FORT": 0.219, "train_start": "2007-01-05", "sampler": "SMOTE", "best_params": "{}", "features": ["NFCI_ret_5d__zrel__EQR_Equity_ret_1d", "heston_var_ev_h5__macross__SRE_Sempra_ret_5d", "US6M_Rate_ret_20d", "VVIX_ret_1d__div__vix_acceleration_1d", "SRE_Sempra_ret_5d__macross__VIX_Price_ret_5d", "NFCI_ret_20d__ret5x__VIX_Price_ret_5d", "heston_var_ev_h5__ret5x__HD_zscore_60d", "NFCI_ret_5d__div__DAX_Germany_vol_20d", "NFCI_ret_5d__ret5x__NFCI_ret_20d", "NFCI_ret_20d__div__DAX_Germany_vol_20d", "Nikkei_Japan_vol_20d", "WTI_Oil_FRED_zscore_60d", "NFCI_ret_5d__minus__VVIX_ret_1d", "NOC_Northrop_ret_1d__macross__vix_acceleration_1d", "NFCI_ret_5d__ret5x__EQR_Equity_ret_1d", "DAX_Germany_vol_20d", "NFCI_ret_5d__zrel__TM_Telephone_ret_5d"], "is_new": false}, {"model_id": "v1_h3_NORMAL_GradientBoosting_N5", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 3, "n_features": 5, "F1_dir": 0.5109, "F1_UP_FORT": 0.3333, "F1_DOWN_FORT": 0.2706, "train_start": "2007-01-05", "sampler": "SMOTE", "best_params": "{}", "features": ["NFCI_ret_5d__zrel__EQR_Equity_ret_1d", "heston_var_ev_h5__macross__SRE_Sempra_ret_5d", "US6M_Rate_ret_20d", "VVIX_ret_1d__div__vix_acceleration_1d", "SRE_Sempra_ret_5d__macross__VIX_Price_ret_5d"], "is_new": false}, {"model_id": "v1_h3_NORMAL_GradientBoosting_N6", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 3, "n_features": 6, "F1_dir": 0.5225, "F1_UP_FORT": 0.2697, "F1_DOWN_FORT": 0.2985, "train_start": "2007-01-05", "sampler": "SMOTE", "best_params": "{}", "features": ["NFCI_ret_5d__zrel__EQR_Equity_ret_1d", "heston_var_ev_h5__macross__SRE_Sempra_ret_5d", "US6M_Rate_ret_20d", "VVIX_ret_1d__div__vix_acceleration_1d", "SRE_Sempra_ret_5d__macross__VIX_Price_ret_5d", "NFCI_ret_20d__ret5x__VIX_Price_ret_5d"], "is_new": false}, {"model_id": "v1_h3_NORMAL_GradientBoosting_N7", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 3, "n_features": 7, "F1_dir": 0.5215, "F1_UP_FORT": 0.2751, "F1_DOWN_FORT": 0.2581, "train_start": "2007-01-05", "sampler": "SMOTE", "best_params": "{}", "features": ["NFCI_ret_5d__zrel__EQR_Equity_ret_1d", "heston_var_ev_h5__macross__SRE_Sempra_ret_5d", "US6M_Rate_ret_20d", "VVIX_ret_1d__div__vix_acceleration_1d", "SRE_Sempra_ret_5d__macross__VIX_Price_ret_5d", "NFCI_ret_20d__ret5x__VIX_Price_ret_5d", "heston_var_ev_h5__ret5x__HD_zscore_60d"], "is_new": false}, {"model_id": "v1_h3_NORMAL_GradientBoosting_N11", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 3, "n_features": 11, "F1_dir": 0.5049, "F1_UP_FORT": 0.2485, "F1_DOWN_FORT": 0.2866, "train_start": "2007-01-05", "sampler": "SMOTE", "best_params": "{}", "features": ["NFCI_ret_5d__zrel__EQR_Equity_ret_1d", "heston_var_ev_h5__macross__SRE_Sempra_ret_5d", "US6M_Rate_ret_20d", "VVIX_ret_1d__div__vix_acceleration_1d", "SRE_Sempra_ret_5d__macross__VIX_Price_ret_5d", "NFCI_ret_20d__ret5x__VIX_Price_ret_5d", "heston_var_ev_h5__ret5x__HD_zscore_60d", "NFCI_ret_5d__div__DAX_Germany_vol_20d", "NFCI_ret_5d__ret5x__NFCI_ret_20d", "NFCI_ret_20d__div__DAX_Germany_vol_20d", "Nikkei_Japan_vol_20d"], "is_new": false}, {"model_id": "v1_h3_NORMAL_GradientBoosting_N12", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 3, "n_features": 12, "F1_dir": 0.5147, "F1_UP_FORT": 0.2933, "F1_DOWN_FORT": 0.25, "train_start": "2007-01-05", "sampler": "SMOTE", "best_params": "{}", "features": ["NFCI_ret_5d__zrel__EQR_Equity_ret_1d", "heston_var_ev_h5__macross__SRE_Sempra_ret_5d", "US6M_Rate_ret_20d", "VVIX_ret_1d__div__vix_acceleration_1d", "SRE_Sempra_ret_5d__macross__VIX_Price_ret_5d", "NFCI_ret_20d__ret5x__VIX_Price_ret_5d", "heston_var_ev_h5__ret5x__HD_zscore_60d", "NFCI_ret_5d__div__DAX_Germany_vol_20d", "NFCI_ret_5d__ret5x__NFCI_ret_20d", "NFCI_ret_20d__div__DAX_Germany_vol_20d", "Nikkei_Japan_vol_20d", "WTI_Oil_FRED_zscore_60d"], "is_new": false}, {"model_id": "v1_h3_NORMAL_GradientBoosting_N13", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 3, "n_features": 13, "F1_dir": 0.5067, "F1_UP_FORT": 0.2921, "F1_DOWN_FORT": 0.2492, "train_start": "2007-01-05", "sampler": "SMOTE", "best_params": "{}", "features": ["NFCI_ret_5d__zrel__EQR_Equity_ret_1d", "heston_var_ev_h5__macross__SRE_Sempra_ret_5d", "US6M_Rate_ret_20d", "VVIX_ret_1d__div__vix_acceleration_1d", "SRE_Sempra_ret_5d__macross__VIX_Price_ret_5d", "NFCI_ret_20d__ret5x__VIX_Price_ret_5d", "heston_var_ev_h5__ret5x__HD_zscore_60d", "NFCI_ret_5d__div__DAX_Germany_vol_20d", "NFCI_ret_5d__ret5x__NFCI_ret_20d", "NFCI_ret_20d__div__DAX_Germany_vol_20d", "Nikkei_Japan_vol_20d", "WTI_Oil_FRED_zscore_60d", "NFCI_ret_5d__minus__VVIX_ret_1d"], "is_new": false}, {"model_id": "v1_h3_NORMAL_GradientBoosting_N14", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 3, "n_features": 14, "F1_dir": 0.5164, "F1_UP_FORT": 0.2904, "F1_DOWN_FORT": 0.2045, "train_start": "2007-01-05", "sampler": "SMOTE", "best_params": "{}", "features": ["NFCI_ret_5d__zrel__EQR_Equity_ret_1d", "heston_var_ev_h5__macross__SRE_Sempra_ret_5d", "US6M_Rate_ret_20d", "VVIX_ret_1d__div__vix_acceleration_1d", "SRE_Sempra_ret_5d__macross__VIX_Price_ret_5d", "NFCI_ret_20d__ret5x__VIX_Price_ret_5d", "heston_var_ev_h5__ret5x__HD_zscore_60d", "NFCI_ret_5d__div__DAX_Germany_vol_20d", "NFCI_ret_5d__ret5x__NFCI_ret_20d", "NFCI_ret_20d__div__DAX_Germany_vol_20d", "Nikkei_Japan_vol_20d", "WTI_Oil_FRED_zscore_60d", "NFCI_ret_5d__minus__VVIX_ret_1d", "NOC_Northrop_ret_1d__macross__vix_acceleration_1d"], "is_new": false}, {"model_id": "v1_h3_NORMAL_GradientBoosting_N15", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 3, "n_features": 15, "F1_dir": 0.5322, "F1_UP_FORT": 0.2857, "F1_DOWN_FORT": 0.2545, "train_start": "2007-01-05", "sampler": "SMOTE", "best_params": "{}", "features": ["NFCI_ret_5d__zrel__EQR_Equity_ret_1d", "heston_var_ev_h5__macross__SRE_Sempra_ret_5d", "US6M_Rate_ret_20d", "VVIX_ret_1d__div__vix_acceleration_1d", "SRE_Sempra_ret_5d__macross__VIX_Price_ret_5d", "NFCI_ret_20d__ret5x__VIX_Price_ret_5d", "heston_var_ev_h5__ret5x__HD_zscore_60d", "NFCI_ret_5d__div__DAX_Germany_vol_20d", "NFCI_ret_5d__ret5x__NFCI_ret_20d", "NFCI_ret_20d__div__DAX_Germany_vol_20d", "Nikkei_Japan_vol_20d", "WTI_Oil_FRED_zscore_60d", "NFCI_ret_5d__minus__VVIX_ret_1d", "NOC_Northrop_ret_1d__macross__vix_acceleration_1d", "NFCI_ret_5d__ret5x__EQR_Equity_ret_1d"], "is_new": false}, {"model_id": "v1_h3_NORMAL_GradientBoosting_N16", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 3, "n_features": 16, "F1_dir": 0.5277, "F1_UP_FORT": 0.3144, "F1_DOWN_FORT": 0.2419, "train_start": "2007-01-05", "sampler": "SMOTE", "best_params": "{}", "features": ["NFCI_ret_5d__zrel__EQR_Equity_ret_1d", "heston_var_ev_h5__macross__SRE_Sempra_ret_5d", "US6M_Rate_ret_20d", "VVIX_ret_1d__div__vix_acceleration_1d", "SRE_Sempra_ret_5d__macross__VIX_Price_ret_5d", "NFCI_ret_20d__ret5x__VIX_Price_ret_5d", "heston_var_ev_h5__ret5x__HD_zscore_60d", "NFCI_ret_5d__div__DAX_Germany_vol_20d", "NFCI_ret_5d__ret5x__NFCI_ret_20d", "NFCI_ret_20d__div__DAX_Germany_vol_20d", "Nikkei_Japan_vol_20d", "WTI_Oil_FRED_zscore_60d", "NFCI_ret_5d__minus__VVIX_ret_1d", "NOC_Northrop_ret_1d__macross__vix_acceleration_1d", "NFCI_ret_5d__ret5x__EQR_Equity_ret_1d", "DAX_Germany_vol_20d"], "is_new": false}, {"model_id": "v1_h3_NORMAL_GradientBoosting_N17", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 3, "n_features": 17, "F1_dir": 0.5312, "F1_UP_FORT": 0.3281, "F1_DOWN_FORT": 0.233, "train_start": "2007-01-05", "sampler": "SMOTE", "best_params": "{}", "features": ["NFCI_ret_5d__zrel__EQR_Equity_ret_1d", "heston_var_ev_h5__macross__SRE_Sempra_ret_5d", "US6M_Rate_ret_20d", "VVIX_ret_1d__div__vix_acceleration_1d", "SRE_Sempra_ret_5d__macross__VIX_Price_ret_5d", "NFCI_ret_20d__ret5x__VIX_Price_ret_5d", "heston_var_ev_h5__ret5x__HD_zscore_60d", "NFCI_ret_5d__div__DAX_Germany_vol_20d", "NFCI_ret_5d__ret5x__NFCI_ret_20d", "NFCI_ret_20d__div__DAX_Germany_vol_20d", "Nikkei_Japan_vol_20d", "WTI_Oil_FRED_zscore_60d", "NFCI_ret_5d__minus__VVIX_ret_1d", "NOC_Northrop_ret_1d__macross__vix_acceleration_1d", "NFCI_ret_5d__ret5x__EQR_Equity_ret_1d", "DAX_Germany_vol_20d", "NFCI_ret_5d__zrel__TM_Telephone_ret_5d"], "is_new": false}, {"model_id": "v1_h3_NORMAL_GradientBoosting_N18", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 3, "n_features": 18, "F1_dir": 0.5012, "F1_UP_FORT": 0.2761, "F1_DOWN_FORT": 0.2756, "train_start": "2007-01-05", "sampler": "SMOTE", "best_params": "{}", "features": ["NFCI_ret_5d__zrel__EQR_Equity_ret_1d", "heston_var_ev_h5__macross__SRE_Sempra_ret_5d", "US6M_Rate_ret_20d", "VVIX_ret_1d__div__vix_acceleration_1d", "SRE_Sempra_ret_5d__macross__VIX_Price_ret_5d", "NFCI_ret_20d__ret5x__VIX_Price_ret_5d", "heston_var_ev_h5__ret5x__HD_zscore_60d", "NFCI_ret_5d__div__DAX_Germany_vol_20d", "NFCI_ret_5d__ret5x__NFCI_ret_20d", "NFCI_ret_20d__div__DAX_Germany_vol_20d", "Nikkei_Japan_vol_20d", "WTI_Oil_FRED_zscore_60d", "NFCI_ret_5d__minus__VVIX_ret_1d", "NOC_Northrop_ret_1d__macross__vix_acceleration_1d", "NFCI_ret_5d__ret5x__EQR_Equity_ret_1d", "DAX_Germany_vol_20d", "NFCI_ret_5d__zrel__TM_Telephone_ret_5d", "EFFR_vol_20d"], "is_new": false}, {"model_id": "v1_h3_NORMAL_GradientBoosting_N19", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 3, "n_features": 19, "F1_dir": 0.5043, "F1_UP_FORT": 0.2892, "F1_DOWN_FORT": 0.2831, "train_start": "2007-01-05", "sampler": "SMOTE", "best_params": "{}", "features": ["NFCI_ret_5d__zrel__EQR_Equity_ret_1d", "heston_var_ev_h5__macross__SRE_Sempra_ret_5d", "US6M_Rate_ret_20d", "VVIX_ret_1d__div__vix_acceleration_1d", "SRE_Sempra_ret_5d__macross__VIX_Price_ret_5d", "NFCI_ret_20d__ret5x__VIX_Price_ret_5d", "heston_var_ev_h5__ret5x__HD_zscore_60d", "NFCI_ret_5d__div__DAX_Germany_vol_20d", "NFCI_ret_5d__ret5x__NFCI_ret_20d", "NFCI_ret_20d__div__DAX_Germany_vol_20d", "Nikkei_Japan_vol_20d", "WTI_Oil_FRED_zscore_60d", "NFCI_ret_5d__minus__VVIX_ret_1d", "NOC_Northrop_ret_1d__macross__vix_acceleration_1d", "NFCI_ret_5d__ret5x__EQR_Equity_ret_1d", "DAX_Germany_vol_20d", "NFCI_ret_5d__zrel__TM_Telephone_ret_5d", "EFFR_vol_20d", "EWM_Malaysia_vol_20d"], "is_new": false}, {"model_id": "v1_h3_NORMAL_RandomForest_N5", "algo": "RandomForest", "regime": "NORMAL", "horizon": 3, "n_features": 5, "F1_dir": 0.5179, "F1_UP_FORT": 0.3123, "F1_DOWN_FORT": 0.2319, "train_start": "2007-01-05", "sampler": "SMOTE", "best_params": "{}", "features": ["NFCI_ret_5d__zrel__EQR_Equity_ret_1d", "heston_var_ev_h5__macross__SRE_Sempra_ret_5d", "US6M_Rate_ret_20d", "VVIX_ret_1d__div__vix_acceleration_1d", "SRE_Sempra_ret_5d__macross__VIX_Price_ret_5d"], "is_new": false}, {"model_id": "v1_h3_NORMAL_RandomForest_N6", "algo": "RandomForest", "regime": "NORMAL", "horizon": 3, "n_features": 6, "F1_dir": 0.5389, "F1_UP_FORT": 0.2963, "F1_DOWN_FORT": 0.2757, "train_start": "2007-01-05", "sampler": "SMOTE", "best_params": "{}", "features": ["NFCI_ret_5d__zrel__EQR_Equity_ret_1d", "heston_var_ev_h5__macross__SRE_Sempra_ret_5d", "US6M_Rate_ret_20d", "VVIX_ret_1d__div__vix_acceleration_1d", "SRE_Sempra_ret_5d__macross__VIX_Price_ret_5d", "NFCI_ret_20d__ret5x__VIX_Price_ret_5d"], "is_new": false}, {"model_id": "v1_h3_NORMAL_RandomForest_N7", "algo": "RandomForest", "regime": "NORMAL", "horizon": 3, "n_features": 7, "F1_dir": 0.5243, "F1_UP_FORT": 0.2882, "F1_DOWN_FORT": 0.2825, "train_start": "2007-01-05", "sampler": "SMOTE", "best_params": "{}", "features": ["NFCI_ret_5d__zrel__EQR_Equity_ret_1d", "heston_var_ev_h5__macross__SRE_Sempra_ret_5d", "US6M_Rate_ret_20d", "VVIX_ret_1d__div__vix_acceleration_1d", "SRE_Sempra_ret_5d__macross__VIX_Price_ret_5d", "NFCI_ret_20d__ret5x__VIX_Price_ret_5d", "heston_var_ev_h5__ret5x__HD_zscore_60d"], "is_new": false}, {"model_id": "v1_h3_NORMAL_RandomForest_N8", "algo": "RandomForest", "regime": "NORMAL", "horizon": 3, "n_features": 8, "F1_dir": 0.5105, "F1_UP_FORT": 0.2798, "F1_DOWN_FORT": 0.2827, "train_start": "2007-01-05", "sampler": "SMOTE", "best_params": "{}", "features": ["NFCI_ret_5d__zrel__EQR_Equity_ret_1d", "heston_var_ev_h5__macross__SRE_Sempra_ret_5d", "US6M_Rate_ret_20d", "VVIX_ret_1d__div__vix_acceleration_1d", "SRE_Sempra_ret_5d__macross__VIX_Price_ret_5d", "NFCI_ret_20d__ret5x__VIX_Price_ret_5d", "heston_var_ev_h5__ret5x__HD_zscore_60d", "NFCI_ret_5d__div__DAX_Germany_vol_20d"], "is_new": false}, {"model_id": "v1_h3_NORMAL_RandomForest_N9", "algo": "RandomForest", "regime": "NORMAL", "horizon": 3, "n_features": 9, "F1_dir": 0.5218, "F1_UP_FORT": 0.2954, "F1_DOWN_FORT": 0.2771, "train_start": "2007-01-05", "sampler": "SMOTE", "best_params": "{}", "features": ["NFCI_ret_5d__zrel__EQR_Equity_ret_1d", "heston_var_ev_h5__macross__SRE_Sempra_ret_5d", "US6M_Rate_ret_20d", "VVIX_ret_1d__div__vix_acceleration_1d", "SRE_Sempra_ret_5d__macross__VIX_Price_ret_5d", "NFCI_ret_20d__ret5x__VIX_Price_ret_5d", "heston_var_ev_h5__ret5x__HD_zscore_60d", "NFCI_ret_5d__div__DAX_Germany_vol_20d", "NFCI_ret_5d__ret5x__NFCI_ret_20d"], "is_new": false}, {"model_id": "v1_h3_NORMAL_RandomForest_N11", "algo": "RandomForest", "regime": "NORMAL", "horizon": 3, "n_features": 11, "F1_dir": 0.508, "F1_UP_FORT": 0.2663, "F1_DOWN_FORT": 0.2755, "train_start": "2007-01-05", "sampler": "SMOTE", "best_params": "{}", "features": ["NFCI_ret_5d__zrel__EQR_Equity_ret_1d", "heston_var_ev_h5__macross__SRE_Sempra_ret_5d", "US6M_Rate_ret_20d", "VVIX_ret_1d__div__vix_acceleration_1d", "SRE_Sempra_ret_5d__macross__VIX_Price_ret_5d", "NFCI_ret_20d__ret5x__VIX_Price_ret_5d", "heston_var_ev_h5__ret5x__HD_zscore_60d", "NFCI_ret_5d__div__DAX_Germany_vol_20d", "NFCI_ret_5d__ret5x__NFCI_ret_20d", "NFCI_ret_20d__div__DAX_Germany_vol_20d", "Nikkei_Japan_vol_20d"], "is_new": false}, {"model_id": "v1_h3_NORMAL_RandomForest_N12", "algo": "RandomForest", "regime": "NORMAL", "horizon": 3, "n_features": 12, "F1_dir": 0.5263, "F1_UP_FORT": 0.2585, "F1_DOWN_FORT": 0.2624, "train_start": "2007-01-05", "sampler": "SMOTE", "best_params": "{}", "features": ["NFCI_ret_5d__zrel__EQR_Equity_ret_1d", "heston_var_ev_h5__macross__SRE_Sempra_ret_5d", "US6M_Rate_ret_20d", "VVIX_ret_1d__div__vix_acceleration_1d", "SRE_Sempra_ret_5d__macross__VIX_Price_ret_5d", "NFCI_ret_20d__ret5x__VIX_Price_ret_5d", "heston_var_ev_h5__ret5x__HD_zscore_60d", "NFCI_ret_5d__div__DAX_Germany_vol_20d", "NFCI_ret_5d__ret5x__NFCI_ret_20d", "NFCI_ret_20d__div__DAX_Germany_vol_20d", "Nikkei_Japan_vol_20d", "WTI_Oil_FRED_zscore_60d"], "is_new": false}, {"model_id": "v1_h3_NORMAL_RandomForest_N13", "algo": "RandomForest", "regime": "NORMAL", "horizon": 3, "n_features": 13, "F1_dir": 0.5019, "F1_UP_FORT": 0.2866, "F1_DOWN_FORT": 0.2283, "train_start": "2007-01-05", "sampler": "SMOTE", "best_params": "{}", "features": ["NFCI_ret_5d__zrel__EQR_Equity_ret_1d", "heston_var_ev_h5__macross__SRE_Sempra_ret_5d", "US6M_Rate_ret_20d", "VVIX_ret_1d__div__vix_acceleration_1d", "SRE_Sempra_ret_5d__macross__VIX_Price_ret_5d", "NFCI_ret_20d__ret5x__VIX_Price_ret_5d", "heston_var_ev_h5__ret5x__HD_zscore_60d", "NFCI_ret_5d__div__DAX_Germany_vol_20d", "NFCI_ret_5d__ret5x__NFCI_ret_20d", "NFCI_ret_20d__div__DAX_Germany_vol_20d", "Nikkei_Japan_vol_20d", "WTI_Oil_FRED_zscore_60d", "NFCI_ret_5d__minus__VVIX_ret_1d"], "is_new": false}, {"model_id": "v1_h3_NORMAL_RandomForest_N14", "algo": "RandomForest", "regime": "NORMAL", "horizon": 3, "n_features": 14, "F1_dir": 0.512, "F1_UP_FORT": 0.2724, "F1_DOWN_FORT": 0.2475, "train_start": "2007-01-05", "sampler": "SMOTE", "best_params": "{}", "features": ["NFCI_ret_5d__zrel__EQR_Equity_ret_1d", "heston_var_ev_h5__macross__SRE_Sempra_ret_5d", "US6M_Rate_ret_20d", "VVIX_ret_1d__div__vix_acceleration_1d", "SRE_Sempra_ret_5d__macross__VIX_Price_ret_5d", "NFCI_ret_20d__ret5x__VIX_Price_ret_5d", "heston_var_ev_h5__ret5x__HD_zscore_60d", "NFCI_ret_5d__div__DAX_Germany_vol_20d", "NFCI_ret_5d__ret5x__NFCI_ret_20d", "NFCI_ret_20d__div__DAX_Germany_vol_20d", "Nikkei_Japan_vol_20d", "WTI_Oil_FRED_zscore_60d", "NFCI_ret_5d__minus__VVIX_ret_1d", "NOC_Northrop_ret_1d__macross__vix_acceleration_1d"], "is_new": false}, {"model_id": "v1_h3_NORMAL_RandomForest_N15", "algo": "RandomForest", "regime": "NORMAL", "horizon": 3, "n_features": 15, "F1_dir": 0.5057, "F1_UP_FORT": 0.2759, "F1_DOWN_FORT": 0.2567, "train_start": "2007-01-05", "sampler": "SMOTE", "best_params": "{}", "features": ["NFCI_ret_5d__zrel__EQR_Equity_ret_1d", "heston_var_ev_h5__macross__SRE_Sempra_ret_5d", "US6M_Rate_ret_20d", "VVIX_ret_1d__div__vix_acceleration_1d", "SRE_Sempra_ret_5d__macross__VIX_Price_ret_5d", "NFCI_ret_20d__ret5x__VIX_Price_ret_5d", "heston_var_ev_h5__ret5x__HD_zscore_60d", "NFCI_ret_5d__div__DAX_Germany_vol_20d", "NFCI_ret_5d__ret5x__NFCI_ret_20d", "NFCI_ret_20d__div__DAX_Germany_vol_20d", "Nikkei_Japan_vol_20d", "WTI_Oil_FRED_zscore_60d", "NFCI_ret_5d__minus__VVIX_ret_1d", "NOC_Northrop_ret_1d__macross__vix_acceleration_1d", "NFCI_ret_5d__ret5x__EQR_Equity_ret_1d"], "is_new": false}, {"model_id": "v1_h3_NORMAL_RandomForest_N17", "algo": "RandomForest", "regime": "NORMAL", "horizon": 3, "n_features": 17, "F1_dir": 0.5286, "F1_UP_FORT": 0.263, "F1_DOWN_FORT": 0.2431, "train_start": "2007-01-05", "sampler": "SMOTE", "best_params": "{}", "features": ["NFCI_ret_5d__zrel__EQR_Equity_ret_1d", "heston_var_ev_h5__macross__SRE_Sempra_ret_5d", "US6M_Rate_ret_20d", "VVIX_ret_1d__div__vix_acceleration_1d", "SRE_Sempra_ret_5d__macross__VIX_Price_ret_5d", "NFCI_ret_20d__ret5x__VIX_Price_ret_5d", "heston_var_ev_h5__ret5x__HD_zscore_60d", "NFCI_ret_5d__div__DAX_Germany_vol_20d", "NFCI_ret_5d__ret5x__NFCI_ret_20d", "NFCI_ret_20d__div__DAX_Germany_vol_20d", "Nikkei_Japan_vol_20d", "WTI_Oil_FRED_zscore_60d", "NFCI_ret_5d__minus__VVIX_ret_1d", "NOC_Northrop_ret_1d__macross__vix_acceleration_1d", "NFCI_ret_5d__ret5x__EQR_Equity_ret_1d", "DAX_Germany_vol_20d", "NFCI_ret_5d__zrel__TM_Telephone_ret_5d"], "is_new": false}, {"model_id": "v1_h3_NORMAL_RandomForest_N18", "algo": "RandomForest", "regime": "NORMAL", "horizon": 3, "n_features": 18, "F1_dir": 0.5057, "F1_UP_FORT": 0.206, "F1_DOWN_FORT": 0.3015, "train_start": "2007-01-05", "sampler": "SMOTE", "best_params": "{}", "features": ["NFCI_ret_5d__zrel__EQR_Equity_ret_1d", "heston_var_ev_h5__macross__SRE_Sempra_ret_5d", "US6M_Rate_ret_20d", "VVIX_ret_1d__div__vix_acceleration_1d", "SRE_Sempra_ret_5d__macross__VIX_Price_ret_5d", "NFCI_ret_20d__ret5x__VIX_Price_ret_5d", "heston_var_ev_h5__ret5x__HD_zscore_60d", "NFCI_ret_5d__div__DAX_Germany_vol_20d", "NFCI_ret_5d__ret5x__NFCI_ret_20d", "NFCI_ret_20d__div__DAX_Germany_vol_20d", "Nikkei_Japan_vol_20d", "WTI_Oil_FRED_zscore_60d", "NFCI_ret_5d__minus__VVIX_ret_1d", "NOC_Northrop_ret_1d__macross__vix_acceleration_1d", "NFCI_ret_5d__ret5x__EQR_Equity_ret_1d", "DAX_Germany_vol_20d", "NFCI_ret_5d__zrel__TM_Telephone_ret_5d", "EFFR_vol_20d"], "is_new": false}, {"model_id": "v1_h3_NORMAL_LogisticRegression_N5", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 3, "n_features": 5, "F1_dir": 0.5287, "F1_UP_FORT": 0.3514, "F1_DOWN_FORT": 0.1322, "train_start": "2007-01-05", "sampler": "SMOTE", "best_params": "{}", "features": ["NFCI_ret_5d__zrel__EQR_Equity_ret_1d", "heston_var_ev_h5__macross__SRE_Sempra_ret_5d", "US6M_Rate_ret_20d", "VVIX_ret_1d__div__vix_acceleration_1d", "SRE_Sempra_ret_5d__macross__VIX_Price_ret_5d"], "is_new": false}, {"model_id": "v1_h3_NORMAL_LogisticRegression_N6", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 3, "n_features": 6, "F1_dir": 0.5095, "F1_UP_FORT": 0.3509, "F1_DOWN_FORT": 0.102, "train_start": "2007-01-05", "sampler": "SMOTE", "best_params": "{}", "features": ["NFCI_ret_5d__zrel__EQR_Equity_ret_1d", "heston_var_ev_h5__macross__SRE_Sempra_ret_5d", "US6M_Rate_ret_20d", "VVIX_ret_1d__div__vix_acceleration_1d", "SRE_Sempra_ret_5d__macross__VIX_Price_ret_5d", "NFCI_ret_20d__ret5x__VIX_Price_ret_5d"], "is_new": false}, {"model_id": "v1_h3_NORMAL_LogisticRegression_N7", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 3, "n_features": 7, "F1_dir": 0.508, "F1_UP_FORT": 0.3509, "F1_DOWN_FORT": 0.0938, "train_start": "2007-01-05", "sampler": "SMOTE", "best_params": "{}", "features": ["NFCI_ret_5d__zrel__EQR_Equity_ret_1d", "heston_var_ev_h5__macross__SRE_Sempra_ret_5d", "US6M_Rate_ret_20d", "VVIX_ret_1d__div__vix_acceleration_1d", "SRE_Sempra_ret_5d__macross__VIX_Price_ret_5d", "NFCI_ret_20d__ret5x__VIX_Price_ret_5d", "heston_var_ev_h5__ret5x__HD_zscore_60d"], "is_new": false}, {"model_id": "v1_h3_NORMAL_LogisticRegression_N8", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 3, "n_features": 8, "F1_dir": 0.5152, "F1_UP_FORT": 0.3511, "F1_DOWN_FORT": 0.1065, "train_start": "2007-01-05", "sampler": "SMOTE", "best_params": "{}", "features": ["NFCI_ret_5d__zrel__EQR_Equity_ret_1d", "heston_var_ev_h5__macross__SRE_Sempra_ret_5d", "US6M_Rate_ret_20d", "VVIX_ret_1d__div__vix_acceleration_1d", "SRE_Sempra_ret_5d__macross__VIX_Price_ret_5d", "NFCI_ret_20d__ret5x__VIX_Price_ret_5d", "heston_var_ev_h5__ret5x__HD_zscore_60d", "NFCI_ret_5d__div__DAX_Germany_vol_20d"], "is_new": false}, {"model_id": "v1_h3_NORMAL_LogisticRegression_N9", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 3, "n_features": 9, "F1_dir": 0.5082, "F1_UP_FORT": 0.3624, "F1_DOWN_FORT": 0.1119, "train_start": "2007-01-05", "sampler": "SMOTE", "best_params": "{}", "features": ["NFCI_ret_5d__zrel__EQR_Equity_ret_1d", "heston_var_ev_h5__macross__SRE_Sempra_ret_5d", "US6M_Rate_ret_20d", "VVIX_ret_1d__div__vix_acceleration_1d", "SRE_Sempra_ret_5d__macross__VIX_Price_ret_5d", "NFCI_ret_20d__ret5x__VIX_Price_ret_5d", "heston_var_ev_h5__ret5x__HD_zscore_60d", "NFCI_ret_5d__div__DAX_Germany_vol_20d", "NFCI_ret_5d__ret5x__NFCI_ret_20d"], "is_new": false}, {"model_id": "v1_h3_NORMAL_LogisticRegression_N10", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 3, "n_features": 10, "F1_dir": 0.5083, "F1_UP_FORT": 0.3583, "F1_DOWN_FORT": 0.1484, "train_start": "2007-01-05", "sampler": "SMOTE", "best_params": "{}", "features": ["NFCI_ret_5d__zrel__EQR_Equity_ret_1d", "heston_var_ev_h5__macross__SRE_Sempra_ret_5d", "US6M_Rate_ret_20d", "VVIX_ret_1d__div__vix_acceleration_1d", "SRE_Sempra_ret_5d__macross__VIX_Price_ret_5d", "NFCI_ret_20d__ret5x__VIX_Price_ret_5d", "heston_var_ev_h5__ret5x__HD_zscore_60d", "NFCI_ret_5d__div__DAX_Germany_vol_20d", "NFCI_ret_5d__ret5x__NFCI_ret_20d", "NFCI_ret_20d__div__DAX_Germany_vol_20d"], "is_new": false}, {"model_id": "v1_h3_NORMAL_LogisticRegression_N11", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 3, "n_features": 11, "F1_dir": 0.5245, "F1_UP_FORT": 0.3636, "F1_DOWN_FORT": 0.1544, "train_start": "2007-01-05", "sampler": "SMOTE", "best_params": "{}", "features": ["NFCI_ret_5d__zrel__EQR_Equity_ret_1d", "heston_var_ev_h5__macross__SRE_Sempra_ret_5d", "US6M_Rate_ret_20d", "VVIX_ret_1d__div__vix_acceleration_1d", "SRE_Sempra_ret_5d__macross__VIX_Price_ret_5d", "NFCI_ret_20d__ret5x__VIX_Price_ret_5d", "heston_var_ev_h5__ret5x__HD_zscore_60d", "NFCI_ret_5d__div__DAX_Germany_vol_20d", "NFCI_ret_5d__ret5x__NFCI_ret_20d", "NFCI_ret_20d__div__DAX_Germany_vol_20d", "Nikkei_Japan_vol_20d"], "is_new": false}, {"model_id": "v1_h3_NORMAL_LogisticRegression_N12", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 3, "n_features": 12, "F1_dir": 0.5139, "F1_UP_FORT": 0.3342, "F1_DOWN_FORT": 0.2244, "train_start": "2007-01-05", "sampler": "SMOTE", "best_params": "{}", "features": ["NFCI_ret_5d__zrel__EQR_Equity_ret_1d", "heston_var_ev_h5__macross__SRE_Sempra_ret_5d", "US6M_Rate_ret_20d", "VVIX_ret_1d__div__vix_acceleration_1d", "SRE_Sempra_ret_5d__macross__VIX_Price_ret_5d", "NFCI_ret_20d__ret5x__VIX_Price_ret_5d", "heston_var_ev_h5__ret5x__HD_zscore_60d", "NFCI_ret_5d__div__DAX_Germany_vol_20d", "NFCI_ret_5d__ret5x__NFCI_ret_20d", "NFCI_ret_20d__div__DAX_Germany_vol_20d", "Nikkei_Japan_vol_20d", "WTI_Oil_FRED_zscore_60d"], "is_new": false}, {"model_id": "v1_h3_NORMAL_LogisticRegression_N13", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 3, "n_features": 13, "F1_dir": 0.5208, "F1_UP_FORT": 0.3073, "F1_DOWN_FORT": 0.2375, "train_start": "2007-01-05", "sampler": "SMOTE", "best_params": "{}", "features": ["NFCI_ret_5d__zrel__EQR_Equity_ret_1d", "heston_var_ev_h5__macross__SRE_Sempra_ret_5d", "US6M_Rate_ret_20d", "VVIX_ret_1d__div__vix_acceleration_1d", "SRE_Sempra_ret_5d__macross__VIX_Price_ret_5d", "NFCI_ret_20d__ret5x__VIX_Price_ret_5d", "heston_var_ev_h5__ret5x__HD_zscore_60d", "NFCI_ret_5d__div__DAX_Germany_vol_20d", "NFCI_ret_5d__ret5x__NFCI_ret_20d", "NFCI_ret_20d__div__DAX_Germany_vol_20d", "Nikkei_Japan_vol_20d", "WTI_Oil_FRED_zscore_60d", "NFCI_ret_5d__minus__VVIX_ret_1d"], "is_new": false}, {"model_id": "v1_h3_NORMAL_LogisticRegression_N14", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 3, "n_features": 14, "F1_dir": 0.5262, "F1_UP_FORT": 0.3045, "F1_DOWN_FORT": 0.2407, "train_start": "2007-01-05", "sampler": "SMOTE", "best_params": "{}", "features": ["NFCI_ret_5d__zrel__EQR_Equity_ret_1d", "heston_var_ev_h5__macross__SRE_Sempra_ret_5d", "US6M_Rate_ret_20d", "VVIX_ret_1d__div__vix_acceleration_1d", "SRE_Sempra_ret_5d__macross__VIX_Price_ret_5d", "NFCI_ret_20d__ret5x__VIX_Price_ret_5d", "heston_var_ev_h5__ret5x__HD_zscore_60d", "NFCI_ret_5d__div__DAX_Germany_vol_20d", "NFCI_ret_5d__ret5x__NFCI_ret_20d", "NFCI_ret_20d__div__DAX_Germany_vol_20d", "Nikkei_Japan_vol_20d", "WTI_Oil_FRED_zscore_60d", "NFCI_ret_5d__minus__VVIX_ret_1d", "NOC_Northrop_ret_1d__macross__vix_acceleration_1d"], "is_new": false}, {"model_id": "v1_h3_NORMAL_LogisticRegression_N15", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 3, "n_features": 15, "F1_dir": 0.5262, "F1_UP_FORT": 0.2982, "F1_DOWN_FORT": 0.2462, "train_start": "2007-01-05", "sampler": "SMOTE", "best_params": "{}", "features": ["NFCI_ret_5d__zrel__EQR_Equity_ret_1d", "heston_var_ev_h5__macross__SRE_Sempra_ret_5d", "US6M_Rate_ret_20d", "VVIX_ret_1d__div__vix_acceleration_1d", "SRE_Sempra_ret_5d__macross__VIX_Price_ret_5d", "NFCI_ret_20d__ret5x__VIX_Price_ret_5d", "heston_var_ev_h5__ret5x__HD_zscore_60d", "NFCI_ret_5d__div__DAX_Germany_vol_20d", "NFCI_ret_5d__ret5x__NFCI_ret_20d", "NFCI_ret_20d__div__DAX_Germany_vol_20d", "Nikkei_Japan_vol_20d", "WTI_Oil_FRED_zscore_60d", "NFCI_ret_5d__minus__VVIX_ret_1d", "NOC_Northrop_ret_1d__macross__vix_acceleration_1d", "NFCI_ret_5d__ret5x__EQR_Equity_ret_1d"], "is_new": false}, {"model_id": "v1_h3_NORMAL_LogisticRegression_N16", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 3, "n_features": 16, "F1_dir": 0.5086, "F1_UP_FORT": 0.314, "F1_DOWN_FORT": 0.1474, "train_start": "2007-01-05", "sampler": "SMOTE", "best_params": "{}", "features": ["NFCI_ret_5d__zrel__EQR_Equity_ret_1d", "heston_var_ev_h5__macross__SRE_Sempra_ret_5d", "US6M_Rate_ret_20d", "VVIX_ret_1d__div__vix_acceleration_1d", "SRE_Sempra_ret_5d__macross__VIX_Price_ret_5d", "NFCI_ret_20d__ret5x__VIX_Price_ret_5d", "heston_var_ev_h5__ret5x__HD_zscore_60d", "NFCI_ret_5d__div__DAX_Germany_vol_20d", "NFCI_ret_5d__ret5x__NFCI_ret_20d", "NFCI_ret_20d__div__DAX_Germany_vol_20d", "Nikkei_Japan_vol_20d", "WTI_Oil_FRED_zscore_60d", "NFCI_ret_5d__minus__VVIX_ret_1d", "NOC_Northrop_ret_1d__macross__vix_acceleration_1d", "NFCI_ret_5d__ret5x__EQR_Equity_ret_1d", "DAX_Germany_vol_20d"], "is_new": false}, {"model_id": "v1_h3_NORMAL_LogisticRegression_N17", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 3, "n_features": 17, "F1_dir": 0.5076, "F1_UP_FORT": 0.3103, "F1_DOWN_FORT": 0.1712, "train_start": "2007-01-05", "sampler": "SMOTE", "best_params": "{}", "features": ["NFCI_ret_5d__zrel__EQR_Equity_ret_1d", "heston_var_ev_h5__macross__SRE_Sempra_ret_5d", "US6M_Rate_ret_20d", "VVIX_ret_1d__div__vix_acceleration_1d", "SRE_Sempra_ret_5d__macross__VIX_Price_ret_5d", "NFCI_ret_20d__ret5x__VIX_Price_ret_5d", "heston_var_ev_h5__ret5x__HD_zscore_60d", "NFCI_ret_5d__div__DAX_Germany_vol_20d", "NFCI_ret_5d__ret5x__NFCI_ret_20d", "NFCI_ret_20d__div__DAX_Germany_vol_20d", "Nikkei_Japan_vol_20d", "WTI_Oil_FRED_zscore_60d", "NFCI_ret_5d__minus__VVIX_ret_1d", "NOC_Northrop_ret_1d__macross__vix_acceleration_1d", "NFCI_ret_5d__ret5x__EQR_Equity_ret_1d", "DAX_Germany_vol_20d", "NFCI_ret_5d__zrel__TM_Telephone_ret_5d"], "is_new": false}, {"model_id": "v1_h3_NORMAL_LogisticRegression_N18", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 3, "n_features": 18, "F1_dir": 0.501, "F1_UP_FORT": 0.2903, "F1_DOWN_FORT": 0.1877, "train_start": "2007-01-05", "sampler": "SMOTE", "best_params": "{}", "features": ["NFCI_ret_5d__zrel__EQR_Equity_ret_1d", "heston_var_ev_h5__macross__SRE_Sempra_ret_5d", "US6M_Rate_ret_20d", "VVIX_ret_1d__div__vix_acceleration_1d", "SRE_Sempra_ret_5d__macross__VIX_Price_ret_5d", "NFCI_ret_20d__ret5x__VIX_Price_ret_5d", "heston_var_ev_h5__ret5x__HD_zscore_60d", "NFCI_ret_5d__div__DAX_Germany_vol_20d", "NFCI_ret_5d__ret5x__NFCI_ret_20d", "NFCI_ret_20d__div__DAX_Germany_vol_20d", "Nikkei_Japan_vol_20d", "WTI_Oil_FRED_zscore_60d", "NFCI_ret_5d__minus__VVIX_ret_1d", "NOC_Northrop_ret_1d__macross__vix_acceleration_1d", "NFCI_ret_5d__ret5x__EQR_Equity_ret_1d", "DAX_Germany_vol_20d", "NFCI_ret_5d__zrel__TM_Telephone_ret_5d", "EFFR_vol_20d"], "is_new": false}, {"model_id": "v1_h3_NORMAL_LightGBM_Optuna_N8", "algo": "LightGBM", "regime": "NORMAL", "horizon": 3, "n_features": 8, "F1_dir": 0.5121, "F1_UP_FORT": 0.2682, "F1_DOWN_FORT": 0.2969, "train_start": "2007-01-05", "sampler": "SMOTE", "best_params": "{}", "features": ["NFCI_ret_5d__zrel__EQR_Equity_ret_1d", "heston_var_ev_h5__macross__SRE_Sempra_ret_5d", "US6M_Rate_ret_20d", "VVIX_ret_1d__div__vix_acceleration_1d", "SRE_Sempra_ret_5d__macross__VIX_Price_ret_5d", "NFCI_ret_20d__ret5x__VIX_Price_ret_5d", "heston_var_ev_h5__ret5x__HD_zscore_60d", "NFCI_ret_5d__div__DAX_Germany_vol_20d"], "is_new": false}, {"model_id": "v1_h3_NORMAL_LightGBM_OptunaCal_N8", "algo": "LightGBMCal", "regime": "NORMAL", "horizon": 3, "n_features": 8, "F1_dir": 0.5176, "F1_UP_FORT": 0.2849, "F1_DOWN_FORT": 0.2898, "train_start": "2007-01-05", "sampler": "SMOTE", "best_params": "{}", "features": ["NFCI_ret_5d__zrel__EQR_Equity_ret_1d", "heston_var_ev_h5__macross__SRE_Sempra_ret_5d", "US6M_Rate_ret_20d", "VVIX_ret_1d__div__vix_acceleration_1d", "SRE_Sempra_ret_5d__macross__VIX_Price_ret_5d", "NFCI_ret_20d__ret5x__VIX_Price_ret_5d", "heston_var_ev_h5__ret5x__HD_zscore_60d", "NFCI_ret_5d__div__DAX_Germany_vol_20d"], "is_new": false}, {"model_id": "v1_h3_STRESS_XGBoost_N5", "algo": "XGBoost", "regime": "STRESS", "horizon": 3, "n_features": 5, "F1_dir": 0.5234, "F1_UP_FORT": 0.2222, "F1_DOWN_FORT": 0.3895, "train_start": "2000-11-16", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_vs_ma20__macross__vix_zscore_10d", "EFFR_ret_5d__div__BBY_BestBuy_zscore_60d", "vix_mean_abs_ret_5d__zrel__EQIX_Equinix_ret_5d", "US1Y_Rate_ret_20d__zrel__US2Y_Rate_ret_5d", "hmm_p_stress"], "is_new": false}, {"model_id": "v1_h3_STRESS_XGBoost_N6", "algo": "XGBoost", "regime": "STRESS", "horizon": 3, "n_features": 6, "F1_dir": 0.502, "F1_UP_FORT": 0.1765, "F1_DOWN_FORT": 0.3663, "train_start": "2000-11-16", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_vs_ma20__macross__vix_zscore_10d", "EFFR_ret_5d__div__BBY_BestBuy_zscore_60d", "vix_mean_abs_ret_5d__zrel__EQIX_Equinix_ret_5d", "US1Y_Rate_ret_20d__zrel__US2Y_Rate_ret_5d", "hmm_p_stress", "HD_ret_1d"], "is_new": false}, {"model_id": "v1_h3_STRESS_XGBoost_N7", "algo": "XGBoost", "regime": "STRESS", "horizon": 3, "n_features": 7, "F1_dir": 0.5229, "F1_UP_FORT": 0.2055, "F1_DOWN_FORT": 0.4234, "train_start": "2000-11-16", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_vs_ma20__macross__vix_zscore_10d", "EFFR_ret_5d__div__BBY_BestBuy_zscore_60d", "vix_mean_abs_ret_5d__zrel__EQIX_Equinix_ret_5d", "US1Y_Rate_ret_20d__zrel__US2Y_Rate_ret_5d", "hmm_p_stress", "HD_ret_1d", "3M_ret_5d"], "is_new": false}, {"model_id": "v1_h3_STRESS_XGBoost_N8", "algo": "XGBoost", "regime": "STRESS", "horizon": 3, "n_features": 8, "F1_dir": 0.5303, "F1_UP_FORT": 0.2027, "F1_DOWN_FORT": 0.4135, "train_start": "2000-11-16", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_vs_ma20__macross__vix_zscore_10d", "EFFR_ret_5d__div__BBY_BestBuy_zscore_60d", "vix_mean_abs_ret_5d__zrel__EQIX_Equinix_ret_5d", "US1Y_Rate_ret_20d__zrel__US2Y_Rate_ret_5d", "hmm_p_stress", "HD_ret_1d", "3M_ret_5d", "SBUX_ret_5d__div__VRTX_VertexPharm_ret_5d"], "is_new": false}, {"model_id": "v1_h3_STRESS_XGBoost_N9", "algo": "XGBoost", "regime": "STRESS", "horizon": 3, "n_features": 9, "F1_dir": 0.5574, "F1_UP_FORT": 0.2128, "F1_DOWN_FORT": 0.397, "train_start": "2000-11-16", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_vs_ma20__macross__vix_zscore_10d", "EFFR_ret_5d__div__BBY_BestBuy_zscore_60d", "vix_mean_abs_ret_5d__zrel__EQIX_Equinix_ret_5d", "US1Y_Rate_ret_20d__zrel__US2Y_Rate_ret_5d", "hmm_p_stress", "HD_ret_1d", "3M_ret_5d", "SBUX_ret_5d__div__VRTX_VertexPharm_ret_5d", "TED_Spread_zscore_60d"], "is_new": false}, {"model_id": "v1_h3_STRESS_LightGBM_N5", "algo": "LightGBM", "regime": "STRESS", "horizon": 3, "n_features": 5, "F1_dir": 0.5257, "F1_UP_FORT": 0.2105, "F1_DOWN_FORT": 0.3385, "train_start": "2000-11-16", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_vs_ma20__macross__vix_zscore_10d", "EFFR_ret_5d__div__BBY_BestBuy_zscore_60d", "vix_mean_abs_ret_5d__zrel__EQIX_Equinix_ret_5d", "US1Y_Rate_ret_20d__zrel__US2Y_Rate_ret_5d", "hmm_p_stress"], "is_new": false}, {"model_id": "v1_h3_STRESS_LightGBM_N6", "algo": "LightGBM", "regime": "STRESS", "horizon": 3, "n_features": 6, "F1_dir": 0.5102, "F1_UP_FORT": 0.1857, "F1_DOWN_FORT": 0.3308, "train_start": "2000-11-16", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_vs_ma20__macross__vix_zscore_10d", "EFFR_ret_5d__div__BBY_BestBuy_zscore_60d", "vix_mean_abs_ret_5d__zrel__EQIX_Equinix_ret_5d", "US1Y_Rate_ret_20d__zrel__US2Y_Rate_ret_5d", "hmm_p_stress", "HD_ret_1d"], "is_new": false}, {"model_id": "v1_h3_STRESS_LightGBM_N7", "algo": "LightGBM", "regime": "STRESS", "horizon": 3, "n_features": 7, "F1_dir": 0.5109, "F1_UP_FORT": 0.2456, "F1_DOWN_FORT": 0.3735, "train_start": "2000-11-16", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_vs_ma20__macross__vix_zscore_10d", "EFFR_ret_5d__div__BBY_BestBuy_zscore_60d", "vix_mean_abs_ret_5d__zrel__EQIX_Equinix_ret_5d", "US1Y_Rate_ret_20d__zrel__US2Y_Rate_ret_5d", "hmm_p_stress", "HD_ret_1d", "3M_ret_5d"], "is_new": false}, {"model_id": "v1_h3_STRESS_LightGBM_N9", "algo": "LightGBM", "regime": "STRESS", "horizon": 3, "n_features": 9, "F1_dir": 0.5243, "F1_UP_FORT": 0.2177, "F1_DOWN_FORT": 0.3701, "train_start": "2000-11-16", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_vs_ma20__macross__vix_zscore_10d", "EFFR_ret_5d__div__BBY_BestBuy_zscore_60d", "vix_mean_abs_ret_5d__zrel__EQIX_Equinix_ret_5d", "US1Y_Rate_ret_20d__zrel__US2Y_Rate_ret_5d", "hmm_p_stress", "HD_ret_1d", "3M_ret_5d", "SBUX_ret_5d__div__VRTX_VertexPharm_ret_5d", "TED_Spread_zscore_60d"], "is_new": false}, {"model_id": "v1_h3_STRESS_GradientBoosting_N5", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 3, "n_features": 5, "F1_dir": 0.52, "F1_UP_FORT": 0.2286, "F1_DOWN_FORT": 0.3971, "train_start": "2000-11-16", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_vs_ma20__macross__vix_zscore_10d", "EFFR_ret_5d__div__BBY_BestBuy_zscore_60d", "vix_mean_abs_ret_5d__zrel__EQIX_Equinix_ret_5d", "US1Y_Rate_ret_20d__zrel__US2Y_Rate_ret_5d", "hmm_p_stress"], "is_new": false}, {"model_id": "v1_h3_STRESS_GradientBoosting_N6", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 3, "n_features": 6, "F1_dir": 0.507, "F1_UP_FORT": 0.1793, "F1_DOWN_FORT": 0.3658, "train_start": "2000-11-16", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_vs_ma20__macross__vix_zscore_10d", "EFFR_ret_5d__div__BBY_BestBuy_zscore_60d", "vix_mean_abs_ret_5d__zrel__EQIX_Equinix_ret_5d", "US1Y_Rate_ret_20d__zrel__US2Y_Rate_ret_5d", "hmm_p_stress", "HD_ret_1d"], "is_new": false}, {"model_id": "v1_h3_STRESS_GradientBoosting_N7", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 3, "n_features": 7, "F1_dir": 0.5184, "F1_UP_FORT": 0.2308, "F1_DOWN_FORT": 0.426, "train_start": "2000-11-16", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_vs_ma20__macross__vix_zscore_10d", "EFFR_ret_5d__div__BBY_BestBuy_zscore_60d", "vix_mean_abs_ret_5d__zrel__EQIX_Equinix_ret_5d", "US1Y_Rate_ret_20d__zrel__US2Y_Rate_ret_5d", "hmm_p_stress", "HD_ret_1d", "3M_ret_5d"], "is_new": false}, {"model_id": "v1_h3_STRESS_GradientBoosting_N8", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 3, "n_features": 8, "F1_dir": 0.5448, "F1_UP_FORT": 0.2667, "F1_DOWN_FORT": 0.3715, "train_start": "2000-11-16", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_vs_ma20__macross__vix_zscore_10d", "EFFR_ret_5d__div__BBY_BestBuy_zscore_60d", "vix_mean_abs_ret_5d__zrel__EQIX_Equinix_ret_5d", "US1Y_Rate_ret_20d__zrel__US2Y_Rate_ret_5d", "hmm_p_stress", "HD_ret_1d", "3M_ret_5d", "SBUX_ret_5d__div__VRTX_VertexPharm_ret_5d"], "is_new": false}, {"model_id": "v1_h3_STRESS_GradientBoosting_N9", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 3, "n_features": 9, "F1_dir": 0.5376, "F1_UP_FORT": 0.2452, "F1_DOWN_FORT": 0.3871, "train_start": "2000-11-16", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_vs_ma20__macross__vix_zscore_10d", "EFFR_ret_5d__div__BBY_BestBuy_zscore_60d", "vix_mean_abs_ret_5d__zrel__EQIX_Equinix_ret_5d", "US1Y_Rate_ret_20d__zrel__US2Y_Rate_ret_5d", "hmm_p_stress", "HD_ret_1d", "3M_ret_5d", "SBUX_ret_5d__div__VRTX_VertexPharm_ret_5d", "TED_Spread_zscore_60d"], "is_new": false}, {"model_id": "v1_h3_STRESS_RandomForest_N5", "algo": "RandomForest", "regime": "STRESS", "horizon": 3, "n_features": 5, "F1_dir": 0.5275, "F1_UP_FORT": 0.0885, "F1_DOWN_FORT": 0.4459, "train_start": "2000-11-16", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_vs_ma20__macross__vix_zscore_10d", "EFFR_ret_5d__div__BBY_BestBuy_zscore_60d", "vix_mean_abs_ret_5d__zrel__EQIX_Equinix_ret_5d", "US1Y_Rate_ret_20d__zrel__US2Y_Rate_ret_5d", "hmm_p_stress"], "is_new": false}, {"model_id": "v1_h3_STRESS_RandomForest_N6", "algo": "RandomForest", "regime": "STRESS", "horizon": 3, "n_features": 6, "F1_dir": 0.5237, "F1_UP_FORT": 0.1026, "F1_DOWN_FORT": 0.4539, "train_start": "2000-11-16", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_vs_ma20__macross__vix_zscore_10d", "EFFR_ret_5d__div__BBY_BestBuy_zscore_60d", "vix_mean_abs_ret_5d__zrel__EQIX_Equinix_ret_5d", "US1Y_Rate_ret_20d__zrel__US2Y_Rate_ret_5d", "hmm_p_stress", "HD_ret_1d"], "is_new": false}, {"model_id": "v1_h3_STRESS_RandomForest_N7", "algo": "RandomForest", "regime": "STRESS", "horizon": 3, "n_features": 7, "F1_dir": 0.5494, "F1_UP_FORT": 0.1983, "F1_DOWN_FORT": 0.4625, "train_start": "2000-11-16", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_vs_ma20__macross__vix_zscore_10d", "EFFR_ret_5d__div__BBY_BestBuy_zscore_60d", "vix_mean_abs_ret_5d__zrel__EQIX_Equinix_ret_5d", "US1Y_Rate_ret_20d__zrel__US2Y_Rate_ret_5d", "hmm_p_stress", "HD_ret_1d", "3M_ret_5d"], "is_new": false}, {"model_id": "v1_h3_STRESS_RandomForest_N8", "algo": "RandomForest", "regime": "STRESS", "horizon": 3, "n_features": 8, "F1_dir": 0.5408, "F1_UP_FORT": 0.2137, "F1_DOWN_FORT": 0.4615, "train_start": "2000-11-16", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_vs_ma20__macross__vix_zscore_10d", "EFFR_ret_5d__div__BBY_BestBuy_zscore_60d", "vix_mean_abs_ret_5d__zrel__EQIX_Equinix_ret_5d", "US1Y_Rate_ret_20d__zrel__US2Y_Rate_ret_5d", "hmm_p_stress", "HD_ret_1d", "3M_ret_5d", "SBUX_ret_5d__div__VRTX_VertexPharm_ret_5d"], "is_new": false}, {"model_id": "v1_h3_STRESS_RandomForest_N9", "algo": "RandomForest", "regime": "STRESS", "horizon": 3, "n_features": 9, "F1_dir": 0.5507, "F1_UP_FORT": 0.2031, "F1_DOWN_FORT": 0.4564, "train_start": "2000-11-16", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_vs_ma20__macross__vix_zscore_10d", "EFFR_ret_5d__div__BBY_BestBuy_zscore_60d", "vix_mean_abs_ret_5d__zrel__EQIX_Equinix_ret_5d", "US1Y_Rate_ret_20d__zrel__US2Y_Rate_ret_5d", "hmm_p_stress", "HD_ret_1d", "3M_ret_5d", "SBUX_ret_5d__div__VRTX_VertexPharm_ret_5d", "TED_Spread_zscore_60d"], "is_new": false}, {"model_id": "v1_h3_STRESS_LogisticRegression_N5", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 3, "n_features": 5, "F1_dir": 0.567, "F1_UP_FORT": 0.0211, "F1_DOWN_FORT": 0.4709, "train_start": "2000-11-16", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_vs_ma20__macross__vix_zscore_10d", "EFFR_ret_5d__div__BBY_BestBuy_zscore_60d", "vix_mean_abs_ret_5d__zrel__EQIX_Equinix_ret_5d", "US1Y_Rate_ret_20d__zrel__US2Y_Rate_ret_5d", "hmm_p_stress"], "is_new": false}, {"model_id": "v1_h3_STRESS_LogisticRegression_N6", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 3, "n_features": 6, "F1_dir": 0.5403, "F1_UP_FORT": 0.0202, "F1_DOWN_FORT": 0.4706, "train_start": "2000-11-16", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_vs_ma20__macross__vix_zscore_10d", "EFFR_ret_5d__div__BBY_BestBuy_zscore_60d", "vix_mean_abs_ret_5d__zrel__EQIX_Equinix_ret_5d", "US1Y_Rate_ret_20d__zrel__US2Y_Rate_ret_5d", "hmm_p_stress", "HD_ret_1d"], "is_new": false}, {"model_id": "v1_h3_STRESS_LogisticRegression_N7", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 3, "n_features": 7, "F1_dir": 0.5417, "F1_UP_FORT": 0.0202, "F1_DOWN_FORT": 0.483, "train_start": "2000-11-16", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_vs_ma20__macross__vix_zscore_10d", "EFFR_ret_5d__div__BBY_BestBuy_zscore_60d", "vix_mean_abs_ret_5d__zrel__EQIX_Equinix_ret_5d", "US1Y_Rate_ret_20d__zrel__US2Y_Rate_ret_5d", "hmm_p_stress", "HD_ret_1d", "3M_ret_5d"], "is_new": false}, {"model_id": "v1_h3_STRESS_LogisticRegression_N8", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 3, "n_features": 8, "F1_dir": 0.5319, "F1_UP_FORT": 0.1345, "F1_DOWN_FORT": 0.4401, "train_start": "2000-11-16", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_vs_ma20__macross__vix_zscore_10d", "EFFR_ret_5d__div__BBY_BestBuy_zscore_60d", "vix_mean_abs_ret_5d__zrel__EQIX_Equinix_ret_5d", "US1Y_Rate_ret_20d__zrel__US2Y_Rate_ret_5d", "hmm_p_stress", "HD_ret_1d", "3M_ret_5d", "SBUX_ret_5d__div__VRTX_VertexPharm_ret_5d"], "is_new": false}, {"model_id": "v1_h3_STRESS_LogisticRegression_N9", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 3, "n_features": 9, "F1_dir": 0.5188, "F1_UP_FORT": 0.1587, "F1_DOWN_FORT": 0.4207, "train_start": "2000-11-16", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_vs_ma20__macross__vix_zscore_10d", "EFFR_ret_5d__div__BBY_BestBuy_zscore_60d", "vix_mean_abs_ret_5d__zrel__EQIX_Equinix_ret_5d", "US1Y_Rate_ret_20d__zrel__US2Y_Rate_ret_5d", "hmm_p_stress", "HD_ret_1d", "3M_ret_5d", "SBUX_ret_5d__div__VRTX_VertexPharm_ret_5d", "TED_Spread_zscore_60d"], "is_new": false}, {"model_id": "v1_h3_STRESS_GradientBoosting_Optuna_N9", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 3, "n_features": 9, "F1_dir": 0.517, "F1_UP_FORT": 0.1727, "F1_DOWN_FORT": 0.359, "train_start": "2000-11-16", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_vs_ma20__macross__vix_zscore_10d", "EFFR_ret_5d__div__BBY_BestBuy_zscore_60d", "vix_mean_abs_ret_5d__zrel__EQIX_Equinix_ret_5d", "US1Y_Rate_ret_20d__zrel__US2Y_Rate_ret_5d", "hmm_p_stress", "HD_ret_1d", "3M_ret_5d", "SBUX_ret_5d__div__VRTX_VertexPharm_ret_5d", "TED_Spread_zscore_60d"], "is_new": false}, {"model_id": "v1_h3_STRESS_GradientBoosting_OptunaCal_N9", "algo": "GradientBoostingCal", "regime": "STRESS", "horizon": 3, "n_features": 9, "F1_dir": 0.5092, "F1_UP_FORT": 0.1974, "F1_DOWN_FORT": 0.3472, "train_start": "2000-11-16", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_vs_ma20__macross__vix_zscore_10d", "EFFR_ret_5d__div__BBY_BestBuy_zscore_60d", "vix_mean_abs_ret_5d__zrel__EQIX_Equinix_ret_5d", "US1Y_Rate_ret_20d__zrel__US2Y_Rate_ret_5d", "hmm_p_stress", "HD_ret_1d", "3M_ret_5d", "SBUX_ret_5d__div__VRTX_VertexPharm_ret_5d", "TED_Spread_zscore_60d"], "is_new": false}, {"model_id": "v1_h3_GLOBAL_XGBoost_N5", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 3, "n_features": 5, "F1_dir": 0.5852, "F1_UP_FORT": 0.2636, "F1_DOWN_FORT": 0.4403, "train_start": "2001-02-07", "sampler": "SMOTE", "best_params": "{}", "features": ["TXN_vol_20d", "NFCI_ret_5d__div__NFCI_vol_20d", "gjr_condvar_h1", "NFCI_ret_5d", "vix_vs_ma20__zrel__XLB_Materials_zscore_60d"], "is_new": false}, {"model_id": "v1_h3_GLOBAL_XGBoost_N6", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 3, "n_features": 6, "F1_dir": 0.5795, "F1_UP_FORT": 0.2699, "F1_DOWN_FORT": 0.428, "train_start": "2001-02-07", "sampler": "SMOTE", "best_params": "{}", "features": ["TXN_vol_20d", "NFCI_ret_5d__div__NFCI_vol_20d", "gjr_condvar_h1", "NFCI_ret_5d", "vix_vs_ma20__zrel__XLB_Materials_zscore_60d", "vix_zscore_10d__zrel__XLB_Materials_zscore_60d"], "is_new": false}, {"model_id": "v1_h3_GLOBAL_XGBoost_N7", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 3, "n_features": 7, "F1_dir": 0.5636, "F1_UP_FORT": 0.2226, "F1_DOWN_FORT": 0.4185, "train_start": "2001-02-07", "sampler": "SMOTE", "best_params": "{}", "features": ["TXN_vol_20d", "NFCI_ret_5d__div__NFCI_vol_20d", "gjr_condvar_h1", "NFCI_ret_5d", "vix_vs_ma20__zrel__XLB_Materials_zscore_60d", "vix_zscore_10d__zrel__XLB_Materials_zscore_60d", "NWL_Newell_ret_20d__macross__VIX_zscore_60d"], "is_new": false}, {"model_id": "v1_h3_GLOBAL_XGBoost_N8", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 3, "n_features": 8, "F1_dir": 0.5797, "F1_UP_FORT": 0.2473, "F1_DOWN_FORT": 0.4381, "train_start": "2001-02-07", "sampler": "SMOTE", "best_params": "{}", "features": ["TXN_vol_20d", "NFCI_ret_5d__div__NFCI_vol_20d", "gjr_condvar_h1", "NFCI_ret_5d", "vix_vs_ma20__zrel__XLB_Materials_zscore_60d", "vix_zscore_10d__zrel__XLB_Materials_zscore_60d", "NWL_Newell_ret_20d__macross__VIX_zscore_60d", "heston_var_ev_h5"], "is_new": false}, {"model_id": "v1_h3_GLOBAL_XGBoost_N9", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 3, "n_features": 9, "F1_dir": 0.562, "F1_UP_FORT": 0.2411, "F1_DOWN_FORT": 0.4254, "train_start": "2001-02-07", "sampler": "SMOTE", "best_params": "{}", "features": ["TXN_vol_20d", "NFCI_ret_5d__div__NFCI_vol_20d", "gjr_condvar_h1", "NFCI_ret_5d", "vix_vs_ma20__zrel__XLB_Materials_zscore_60d", "vix_zscore_10d__zrel__XLB_Materials_zscore_60d", "NWL_Newell_ret_20d__macross__VIX_zscore_60d", "heston_var_ev_h5", "VIX_Price_ret_5d__minus__US2Y_Rate_ret_5d"], "is_new": false}, {"model_id": "v1_h3_GLOBAL_XGBoost_N10", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 3, "n_features": 10, "F1_dir": 0.5616, "F1_UP_FORT": 0.257, "F1_DOWN_FORT": 0.4255, "train_start": "2001-02-07", "sampler": "SMOTE", "best_params": "{}", "features": ["TXN_vol_20d", "NFCI_ret_5d__div__NFCI_vol_20d", "gjr_condvar_h1", "NFCI_ret_5d", "vix_vs_ma20__zrel__XLB_Materials_zscore_60d", "vix_zscore_10d__zrel__XLB_Materials_zscore_60d", "NWL_Newell_ret_20d__macross__VIX_zscore_60d", "heston_var_ev_h5", "VIX_Price_ret_5d__minus__US2Y_Rate_ret_5d", "Russell_Price_ret_5d__ret5x__HD_zscore_60d"], "is_new": false}, {"model_id": "v1_h3_GLOBAL_XGBoost_N11", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 3, "n_features": 11, "F1_dir": 0.5613, "F1_UP_FORT": 0.2637, "F1_DOWN_FORT": 0.4225, "train_start": "2001-02-07", "sampler": "SMOTE", "best_params": "{}", "features": ["TXN_vol_20d", "NFCI_ret_5d__div__NFCI_vol_20d", "gjr_condvar_h1", "NFCI_ret_5d", "vix_vs_ma20__zrel__XLB_Materials_zscore_60d", "vix_zscore_10d__zrel__XLB_Materials_zscore_60d", "NWL_Newell_ret_20d__macross__VIX_zscore_60d", "heston_var_ev_h5", "VIX_Price_ret_5d__minus__US2Y_Rate_ret_5d", "Russell_Price_ret_5d__ret5x__HD_zscore_60d", "NFCI_ret_5d__minus__CCI_CrownCastle_vol_20d"], "is_new": false}, {"model_id": "v1_h3_GLOBAL_XGBoost_N12", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 3, "n_features": 12, "F1_dir": 0.565, "F1_UP_FORT": 0.2651, "F1_DOWN_FORT": 0.4075, "train_start": "2001-02-07", "sampler": "SMOTE", "best_params": "{}", "features": ["TXN_vol_20d", "NFCI_ret_5d__div__NFCI_vol_20d", "gjr_condvar_h1", "NFCI_ret_5d", "vix_vs_ma20__zrel__XLB_Materials_zscore_60d", "vix_zscore_10d__zrel__XLB_Materials_zscore_60d", "NWL_Newell_ret_20d__macross__VIX_zscore_60d", "heston_var_ev_h5", "VIX_Price_ret_5d__minus__US2Y_Rate_ret_5d", "Russell_Price_ret_5d__ret5x__HD_zscore_60d", "NFCI_ret_5d__minus__CCI_CrownCastle_vol_20d", "CCI_CrownCastle_vol_20d__macross__heston_kappa"], "is_new": false}, {"model_id": "v1_h3_GLOBAL_XGBoost_N13", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 3, "n_features": 13, "F1_dir": 0.555, "F1_UP_FORT": 0.2697, "F1_DOWN_FORT": 0.4108, "train_start": "2001-02-07", "sampler": "SMOTE", "best_params": "{}", "features": ["TXN_vol_20d", "NFCI_ret_5d__div__NFCI_vol_20d", "gjr_condvar_h1", "NFCI_ret_5d", "vix_vs_ma20__zrel__XLB_Materials_zscore_60d", "vix_zscore_10d__zrel__XLB_Materials_zscore_60d", "NWL_Newell_ret_20d__macross__VIX_zscore_60d", "heston_var_ev_h5", "VIX_Price_ret_5d__minus__US2Y_Rate_ret_5d", "Russell_Price_ret_5d__ret5x__HD_zscore_60d", "NFCI_ret_5d__minus__CCI_CrownCastle_vol_20d", "CCI_CrownCastle_vol_20d__macross__heston_kappa", "vix_vs_ma20__div__NWL_Newell_ret_20d"], "is_new": false}, {"model_id": "v1_h3_GLOBAL_XGBoost_N14", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 3, "n_features": 14, "F1_dir": 0.5641, "F1_UP_FORT": 0.2763, "F1_DOWN_FORT": 0.4203, "train_start": "2001-02-07", "sampler": "SMOTE", "best_params": "{}", "features": ["TXN_vol_20d", "NFCI_ret_5d__div__NFCI_vol_20d", "gjr_condvar_h1", "NFCI_ret_5d", "vix_vs_ma20__zrel__XLB_Materials_zscore_60d", "vix_zscore_10d__zrel__XLB_Materials_zscore_60d", "NWL_Newell_ret_20d__macross__VIX_zscore_60d", "heston_var_ev_h5", "VIX_Price_ret_5d__minus__US2Y_Rate_ret_5d", "Russell_Price_ret_5d__ret5x__HD_zscore_60d", "NFCI_ret_5d__minus__CCI_CrownCastle_vol_20d", "CCI_CrownCastle_vol_20d__macross__heston_kappa", "vix_vs_ma20__div__NWL_Newell_ret_20d", "NFCI_vol_20d__ret5x__heston_kappa"], "is_new": false}, {"model_id": "v1_h3_GLOBAL_XGBoost_N15", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 3, "n_features": 15, "F1_dir": 0.5541, "F1_UP_FORT": 0.2764, "F1_DOWN_FORT": 0.4085, "train_start": "2001-02-07", "sampler": "SMOTE", "best_params": "{}", "features": ["TXN_vol_20d", "NFCI_ret_5d__div__NFCI_vol_20d", "gjr_condvar_h1", "NFCI_ret_5d", "vix_vs_ma20__zrel__XLB_Materials_zscore_60d", "vix_zscore_10d__zrel__XLB_Materials_zscore_60d", "NWL_Newell_ret_20d__macross__VIX_zscore_60d", "heston_var_ev_h5", "VIX_Price_ret_5d__minus__US2Y_Rate_ret_5d", "Russell_Price_ret_5d__ret5x__HD_zscore_60d", "NFCI_ret_5d__minus__CCI_CrownCastle_vol_20d", "CCI_CrownCastle_vol_20d__macross__heston_kappa", "vix_vs_ma20__div__NWL_Newell_ret_20d", "NFCI_vol_20d__ret5x__heston_kappa", "NFCI_ret_5d__zrel__Russell_Price_ret_5d"], "is_new": false}, {"model_id": "v1_h3_GLOBAL_XGBoost_N16", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 3, "n_features": 16, "F1_dir": 0.572, "F1_UP_FORT": 0.3028, "F1_DOWN_FORT": 0.409, "train_start": "2001-02-07", "sampler": "SMOTE", "best_params": "{}", "features": ["TXN_vol_20d", "NFCI_ret_5d__div__NFCI_vol_20d", "gjr_condvar_h1", "NFCI_ret_5d", "vix_vs_ma20__zrel__XLB_Materials_zscore_60d", "vix_zscore_10d__zrel__XLB_Materials_zscore_60d", "NWL_Newell_ret_20d__macross__VIX_zscore_60d", "heston_var_ev_h5", "VIX_Price_ret_5d__minus__US2Y_Rate_ret_5d", "Russell_Price_ret_5d__ret5x__HD_zscore_60d", "NFCI_ret_5d__minus__CCI_CrownCastle_vol_20d", "CCI_CrownCastle_vol_20d__macross__heston_kappa", "vix_vs_ma20__div__NWL_Newell_ret_20d", "NFCI_vol_20d__ret5x__heston_kappa", "NFCI_ret_5d__zrel__Russell_Price_ret_5d", "NWL_Newell_ret_20d__ret5x__VIX_zscore_60d"], "is_new": false}, {"model_id": "v1_h3_GLOBAL_XGBoost_N17", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 3, "n_features": 17, "F1_dir": 0.5638, "F1_UP_FORT": 0.2754, "F1_DOWN_FORT": 0.4, "train_start": "2001-02-07", "sampler": "SMOTE", "best_params": "{}", "features": ["TXN_vol_20d", "NFCI_ret_5d__div__NFCI_vol_20d", "gjr_condvar_h1", "NFCI_ret_5d", "vix_vs_ma20__zrel__XLB_Materials_zscore_60d", "vix_zscore_10d__zrel__XLB_Materials_zscore_60d", "NWL_Newell_ret_20d__macross__VIX_zscore_60d", "heston_var_ev_h5", "VIX_Price_ret_5d__minus__US2Y_Rate_ret_5d", "Russell_Price_ret_5d__ret5x__HD_zscore_60d", "NFCI_ret_5d__minus__CCI_CrownCastle_vol_20d", "CCI_CrownCastle_vol_20d__macross__heston_kappa", "vix_vs_ma20__div__NWL_Newell_ret_20d", "NFCI_vol_20d__ret5x__heston_kappa", "NFCI_ret_5d__zrel__Russell_Price_ret_5d", "NWL_Newell_ret_20d__ret5x__VIX_zscore_60d", "BTI_BritishAmerican_ret_5d"], "is_new": false}, {"model_id": "v1_h3_GLOBAL_XGBoost_N18", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 3, "n_features": 18, "F1_dir": 0.5626, "F1_UP_FORT": 0.2892, "F1_DOWN_FORT": 0.4174, "train_start": "2001-02-07", "sampler": "SMOTE", "best_params": "{}", "features": ["TXN_vol_20d", "NFCI_ret_5d__div__NFCI_vol_20d", "gjr_condvar_h1", "NFCI_ret_5d", "vix_vs_ma20__zrel__XLB_Materials_zscore_60d", "vix_zscore_10d__zrel__XLB_Materials_zscore_60d", "NWL_Newell_ret_20d__macross__VIX_zscore_60d", "heston_var_ev_h5", "VIX_Price_ret_5d__minus__US2Y_Rate_ret_5d", "Russell_Price_ret_5d__ret5x__HD_zscore_60d", "NFCI_ret_5d__minus__CCI_CrownCastle_vol_20d", "CCI_CrownCastle_vol_20d__macross__heston_kappa", "vix_vs_ma20__div__NWL_Newell_ret_20d", "NFCI_vol_20d__ret5x__heston_kappa", "NFCI_ret_5d__zrel__Russell_Price_ret_5d", "NWL_Newell_ret_20d__ret5x__VIX_zscore_60d", "BTI_BritishAmerican_ret_5d", "heston_ev_h3"], "is_new": false}, {"model_id": "v1_h3_GLOBAL_LightGBM_N5", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 3, "n_features": 5, "F1_dir": 0.5643, "F1_UP_FORT": 0.2278, "F1_DOWN_FORT": 0.4221, "train_start": "2001-02-07", "sampler": "SMOTE", "best_params": "{}", "features": ["TXN_vol_20d", "NFCI_ret_5d__div__NFCI_vol_20d", "gjr_condvar_h1", "NFCI_ret_5d", "vix_vs_ma20__zrel__XLB_Materials_zscore_60d"], "is_new": false}, {"model_id": "v1_h3_GLOBAL_LightGBM_N6", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 3, "n_features": 6, "F1_dir": 0.5787, "F1_UP_FORT": 0.2466, "F1_DOWN_FORT": 0.4288, "train_start": "2001-02-07", "sampler": "SMOTE", "best_params": "{}", "features": ["TXN_vol_20d", "NFCI_ret_5d__div__NFCI_vol_20d", "gjr_condvar_h1", "NFCI_ret_5d", "vix_vs_ma20__zrel__XLB_Materials_zscore_60d", "vix_zscore_10d__zrel__XLB_Materials_zscore_60d"], "is_new": false}, {"model_id": "v1_h3_GLOBAL_LightGBM_N7", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 3, "n_features": 7, "F1_dir": 0.5686, "F1_UP_FORT": 0.2033, "F1_DOWN_FORT": 0.3903, "train_start": "2001-02-07", "sampler": "SMOTE", "best_params": "{}", "features": ["TXN_vol_20d", "NFCI_ret_5d__div__NFCI_vol_20d", "gjr_condvar_h1", "NFCI_ret_5d", "vix_vs_ma20__zrel__XLB_Materials_zscore_60d", "vix_zscore_10d__zrel__XLB_Materials_zscore_60d", "NWL_Newell_ret_20d__macross__VIX_zscore_60d"], "is_new": false}, {"model_id": "v1_h3_GLOBAL_LightGBM_N8", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 3, "n_features": 8, "F1_dir": 0.5655, "F1_UP_FORT": 0.2329, "F1_DOWN_FORT": 0.4203, "train_start": "2001-02-07", "sampler": "SMOTE", "best_params": "{}", "features": ["TXN_vol_20d", "NFCI_ret_5d__div__NFCI_vol_20d", "gjr_condvar_h1", "NFCI_ret_5d", "vix_vs_ma20__zrel__XLB_Materials_zscore_60d", "vix_zscore_10d__zrel__XLB_Materials_zscore_60d", "NWL_Newell_ret_20d__macross__VIX_zscore_60d", "heston_var_ev_h5"], "is_new": false}, {"model_id": "v1_h3_GLOBAL_LightGBM_N9", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 3, "n_features": 9, "F1_dir": 0.5659, "F1_UP_FORT": 0.237, "F1_DOWN_FORT": 0.4225, "train_start": "2001-02-07", "sampler": "SMOTE", "best_params": "{}", "features": ["TXN_vol_20d", "NFCI_ret_5d__div__NFCI_vol_20d", "gjr_condvar_h1", "NFCI_ret_5d", "vix_vs_ma20__zrel__XLB_Materials_zscore_60d", "vix_zscore_10d__zrel__XLB_Materials_zscore_60d", "NWL_Newell_ret_20d__macross__VIX_zscore_60d", "heston_var_ev_h5", "VIX_Price_ret_5d__minus__US2Y_Rate_ret_5d"], "is_new": false}, {"model_id": "v1_h3_GLOBAL_LightGBM_N10", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 3, "n_features": 10, "F1_dir": 0.5491, "F1_UP_FORT": 0.2443, "F1_DOWN_FORT": 0.3951, "train_start": "2001-02-07", "sampler": "SMOTE", "best_params": "{}", "features": ["TXN_vol_20d", "NFCI_ret_5d__div__NFCI_vol_20d", "gjr_condvar_h1", "NFCI_ret_5d", "vix_vs_ma20__zrel__XLB_Materials_zscore_60d", "vix_zscore_10d__zrel__XLB_Materials_zscore_60d", "NWL_Newell_ret_20d__macross__VIX_zscore_60d", "heston_var_ev_h5", "VIX_Price_ret_5d__minus__US2Y_Rate_ret_5d", "Russell_Price_ret_5d__ret5x__HD_zscore_60d"], "is_new": false}, {"model_id": "v1_h3_GLOBAL_LightGBM_N11", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 3, "n_features": 11, "F1_dir": 0.5655, "F1_UP_FORT": 0.2408, "F1_DOWN_FORT": 0.407, "train_start": "2001-02-07", "sampler": "SMOTE", "best_params": "{}", "features": ["TXN_vol_20d", "NFCI_ret_5d__div__NFCI_vol_20d", "gjr_condvar_h1", "NFCI_ret_5d", "vix_vs_ma20__zrel__XLB_Materials_zscore_60d", "vix_zscore_10d__zrel__XLB_Materials_zscore_60d", "NWL_Newell_ret_20d__macross__VIX_zscore_60d", "heston_var_ev_h5", "VIX_Price_ret_5d__minus__US2Y_Rate_ret_5d", "Russell_Price_ret_5d__ret5x__HD_zscore_60d", "NFCI_ret_5d__minus__CCI_CrownCastle_vol_20d"], "is_new": false}, {"model_id": "v1_h3_GLOBAL_LightGBM_N12", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 3, "n_features": 12, "F1_dir": 0.5709, "F1_UP_FORT": 0.2732, "F1_DOWN_FORT": 0.3964, "train_start": "2001-02-07", "sampler": "SMOTE", "best_params": "{}", "features": ["TXN_vol_20d", "NFCI_ret_5d__div__NFCI_vol_20d", "gjr_condvar_h1", "NFCI_ret_5d", "vix_vs_ma20__zrel__XLB_Materials_zscore_60d", "vix_zscore_10d__zrel__XLB_Materials_zscore_60d", "NWL_Newell_ret_20d__macross__VIX_zscore_60d", "heston_var_ev_h5", "VIX_Price_ret_5d__minus__US2Y_Rate_ret_5d", "Russell_Price_ret_5d__ret5x__HD_zscore_60d", "NFCI_ret_5d__minus__CCI_CrownCastle_vol_20d", "CCI_CrownCastle_vol_20d__macross__heston_kappa"], "is_new": false}, {"model_id": "v1_h3_GLOBAL_LightGBM_N13", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 3, "n_features": 13, "F1_dir": 0.5641, "F1_UP_FORT": 0.2857, "F1_DOWN_FORT": 0.3822, "train_start": "2001-02-07", "sampler": "SMOTE", "best_params": "{}", "features": ["TXN_vol_20d", "NFCI_ret_5d__div__NFCI_vol_20d", "gjr_condvar_h1", "NFCI_ret_5d", "vix_vs_ma20__zrel__XLB_Materials_zscore_60d", "vix_zscore_10d__zrel__XLB_Materials_zscore_60d", "NWL_Newell_ret_20d__macross__VIX_zscore_60d", "heston_var_ev_h5", "VIX_Price_ret_5d__minus__US2Y_Rate_ret_5d", "Russell_Price_ret_5d__ret5x__HD_zscore_60d", "NFCI_ret_5d__minus__CCI_CrownCastle_vol_20d", "CCI_CrownCastle_vol_20d__macross__heston_kappa", "vix_vs_ma20__div__NWL_Newell_ret_20d"], "is_new": false}, {"model_id": "v1_h3_GLOBAL_LightGBM_N14", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 3, "n_features": 14, "F1_dir": 0.5562, "F1_UP_FORT": 0.2834, "F1_DOWN_FORT": 0.3774, "train_start": "2001-02-07", "sampler": "SMOTE", "best_params": "{}", "features": ["TXN_vol_20d", "NFCI_ret_5d__div__NFCI_vol_20d", "gjr_condvar_h1", "NFCI_ret_5d", "vix_vs_ma20__zrel__XLB_Materials_zscore_60d", "vix_zscore_10d__zrel__XLB_Materials_zscore_60d", "NWL_Newell_ret_20d__macross__VIX_zscore_60d", "heston_var_ev_h5", "VIX_Price_ret_5d__minus__US2Y_Rate_ret_5d", "Russell_Price_ret_5d__ret5x__HD_zscore_60d", "NFCI_ret_5d__minus__CCI_CrownCastle_vol_20d", "CCI_CrownCastle_vol_20d__macross__heston_kappa", "vix_vs_ma20__div__NWL_Newell_ret_20d", "NFCI_vol_20d__ret5x__heston_kappa"], "is_new": false}, {"model_id": "v1_h3_GLOBAL_LightGBM_N15", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 3, "n_features": 15, "F1_dir": 0.5654, "F1_UP_FORT": 0.3185, "F1_DOWN_FORT": 0.3835, "train_start": "2001-02-07", "sampler": "SMOTE", "best_params": "{}", "features": ["TXN_vol_20d", "NFCI_ret_5d__div__NFCI_vol_20d", "gjr_condvar_h1", "NFCI_ret_5d", "vix_vs_ma20__zrel__XLB_Materials_zscore_60d", "vix_zscore_10d__zrel__XLB_Materials_zscore_60d", "NWL_Newell_ret_20d__macross__VIX_zscore_60d", "heston_var_ev_h5", "VIX_Price_ret_5d__minus__US2Y_Rate_ret_5d", "Russell_Price_ret_5d__ret5x__HD_zscore_60d", "NFCI_ret_5d__minus__CCI_CrownCastle_vol_20d", "CCI_CrownCastle_vol_20d__macross__heston_kappa", "vix_vs_ma20__div__NWL_Newell_ret_20d", "NFCI_vol_20d__ret5x__heston_kappa", "NFCI_ret_5d__zrel__Russell_Price_ret_5d"], "is_new": false}, {"model_id": "v1_h3_GLOBAL_LightGBM_N16", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 3, "n_features": 16, "F1_dir": 0.5654, "F1_UP_FORT": 0.2622, "F1_DOWN_FORT": 0.4005, "train_start": "2001-02-07", "sampler": "SMOTE", "best_params": "{}", "features": ["TXN_vol_20d", "NFCI_ret_5d__div__NFCI_vol_20d", "gjr_condvar_h1", "NFCI_ret_5d", "vix_vs_ma20__zrel__XLB_Materials_zscore_60d", "vix_zscore_10d__zrel__XLB_Materials_zscore_60d", "NWL_Newell_ret_20d__macross__VIX_zscore_60d", "heston_var_ev_h5", "VIX_Price_ret_5d__minus__US2Y_Rate_ret_5d", "Russell_Price_ret_5d__ret5x__HD_zscore_60d", "NFCI_ret_5d__minus__CCI_CrownCastle_vol_20d", "CCI_CrownCastle_vol_20d__macross__heston_kappa", "vix_vs_ma20__div__NWL_Newell_ret_20d", "NFCI_vol_20d__ret5x__heston_kappa", "NFCI_ret_5d__zrel__Russell_Price_ret_5d", "NWL_Newell_ret_20d__ret5x__VIX_zscore_60d"], "is_new": false}, {"model_id": "v1_h3_GLOBAL_LightGBM_N17", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 3, "n_features": 17, "F1_dir": 0.5585, "F1_UP_FORT": 0.2775, "F1_DOWN_FORT": 0.4074, "train_start": "2001-02-07", "sampler": "SMOTE", "best_params": "{}", "features": ["TXN_vol_20d", "NFCI_ret_5d__div__NFCI_vol_20d", "gjr_condvar_h1", "NFCI_ret_5d", "vix_vs_ma20__zrel__XLB_Materials_zscore_60d", "vix_zscore_10d__zrel__XLB_Materials_zscore_60d", "NWL_Newell_ret_20d__macross__VIX_zscore_60d", "heston_var_ev_h5", "VIX_Price_ret_5d__minus__US2Y_Rate_ret_5d", "Russell_Price_ret_5d__ret5x__HD_zscore_60d", "NFCI_ret_5d__minus__CCI_CrownCastle_vol_20d", "CCI_CrownCastle_vol_20d__macross__heston_kappa", "vix_vs_ma20__div__NWL_Newell_ret_20d", "NFCI_vol_20d__ret5x__heston_kappa", "NFCI_ret_5d__zrel__Russell_Price_ret_5d", "NWL_Newell_ret_20d__ret5x__VIX_zscore_60d", "BTI_BritishAmerican_ret_5d"], "is_new": false}, {"model_id": "v1_h3_GLOBAL_LightGBM_N18", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 3, "n_features": 18, "F1_dir": 0.5674, "F1_UP_FORT": 0.2812, "F1_DOWN_FORT": 0.4212, "train_start": "2001-02-07", "sampler": "SMOTE", "best_params": "{}", "features": ["TXN_vol_20d", "NFCI_ret_5d__div__NFCI_vol_20d", "gjr_condvar_h1", "NFCI_ret_5d", "vix_vs_ma20__zrel__XLB_Materials_zscore_60d", "vix_zscore_10d__zrel__XLB_Materials_zscore_60d", "NWL_Newell_ret_20d__macross__VIX_zscore_60d", "heston_var_ev_h5", "VIX_Price_ret_5d__minus__US2Y_Rate_ret_5d", "Russell_Price_ret_5d__ret5x__HD_zscore_60d", "NFCI_ret_5d__minus__CCI_CrownCastle_vol_20d", "CCI_CrownCastle_vol_20d__macross__heston_kappa", "vix_vs_ma20__div__NWL_Newell_ret_20d", "NFCI_vol_20d__ret5x__heston_kappa", "NFCI_ret_5d__zrel__Russell_Price_ret_5d", "NWL_Newell_ret_20d__ret5x__VIX_zscore_60d", "BTI_BritishAmerican_ret_5d", "heston_ev_h3"], "is_new": false}, {"model_id": "v1_h3_GLOBAL_GradientBoosting_N5", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 3, "n_features": 5, "F1_dir": 0.5825, "F1_UP_FORT": 0.2616, "F1_DOWN_FORT": 0.4382, "train_start": "2001-02-07", "sampler": "SMOTE", "best_params": "{}", "features": ["TXN_vol_20d", "NFCI_ret_5d__div__NFCI_vol_20d", "gjr_condvar_h1", "NFCI_ret_5d", "vix_vs_ma20__zrel__XLB_Materials_zscore_60d"], "is_new": false}, {"model_id": "v1_h3_GLOBAL_GradientBoosting_N6", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 3, "n_features": 6, "F1_dir": 0.5827, "F1_UP_FORT": 0.2715, "F1_DOWN_FORT": 0.4178, "train_start": "2001-02-07", "sampler": "SMOTE", "best_params": "{}", "features": ["TXN_vol_20d", "NFCI_ret_5d__div__NFCI_vol_20d", "gjr_condvar_h1", "NFCI_ret_5d", "vix_vs_ma20__zrel__XLB_Materials_zscore_60d", "vix_zscore_10d__zrel__XLB_Materials_zscore_60d"], "is_new": false}, {"model_id": "v1_h3_GLOBAL_GradientBoosting_N7", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 3, "n_features": 7, "F1_dir": 0.5571, "F1_UP_FORT": 0.2038, "F1_DOWN_FORT": 0.4149, "train_start": "2001-02-07", "sampler": "SMOTE", "best_params": "{}", "features": ["TXN_vol_20d", "NFCI_ret_5d__div__NFCI_vol_20d", "gjr_condvar_h1", "NFCI_ret_5d", "vix_vs_ma20__zrel__XLB_Materials_zscore_60d", "vix_zscore_10d__zrel__XLB_Materials_zscore_60d", "NWL_Newell_ret_20d__macross__VIX_zscore_60d"], "is_new": false}, {"model_id": "v1_h3_GLOBAL_GradientBoosting_N8", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 3, "n_features": 8, "F1_dir": 0.5648, "F1_UP_FORT": 0.2136, "F1_DOWN_FORT": 0.4357, "train_start": "2001-02-07", "sampler": "SMOTE", "best_params": "{}", "features": ["TXN_vol_20d", "NFCI_ret_5d__div__NFCI_vol_20d", "gjr_condvar_h1", "NFCI_ret_5d", "vix_vs_ma20__zrel__XLB_Materials_zscore_60d", "vix_zscore_10d__zrel__XLB_Materials_zscore_60d", "NWL_Newell_ret_20d__macross__VIX_zscore_60d", "heston_var_ev_h5"], "is_new": false}, {"model_id": "v1_h3_GLOBAL_GradientBoosting_N9", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 3, "n_features": 9, "F1_dir": 0.575, "F1_UP_FORT": 0.2351, "F1_DOWN_FORT": 0.4331, "train_start": "2001-02-07", "sampler": "SMOTE", "best_params": "{}", "features": ["TXN_vol_20d", "NFCI_ret_5d__div__NFCI_vol_20d", "gjr_condvar_h1", "NFCI_ret_5d", "vix_vs_ma20__zrel__XLB_Materials_zscore_60d", "vix_zscore_10d__zrel__XLB_Materials_zscore_60d", "NWL_Newell_ret_20d__macross__VIX_zscore_60d", "heston_var_ev_h5", "VIX_Price_ret_5d__minus__US2Y_Rate_ret_5d"], "is_new": false}, {"model_id": "v1_h3_GLOBAL_GradientBoosting_N10", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 3, "n_features": 10, "F1_dir": 0.5537, "F1_UP_FORT": 0.2551, "F1_DOWN_FORT": 0.4104, "train_start": "2001-02-07", "sampler": "SMOTE", "best_params": "{}", "features": ["TXN_vol_20d", "NFCI_ret_5d__div__NFCI_vol_20d", "gjr_condvar_h1", "NFCI_ret_5d", "vix_vs_ma20__zrel__XLB_Materials_zscore_60d", "vix_zscore_10d__zrel__XLB_Materials_zscore_60d", "NWL_Newell_ret_20d__macross__VIX_zscore_60d", "heston_var_ev_h5", "VIX_Price_ret_5d__minus__US2Y_Rate_ret_5d", "Russell_Price_ret_5d__ret5x__HD_zscore_60d"], "is_new": false}, {"model_id": "v1_h3_GLOBAL_GradientBoosting_N11", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 3, "n_features": 11, "F1_dir": 0.5538, "F1_UP_FORT": 0.2517, "F1_DOWN_FORT": 0.4202, "train_start": "2001-02-07", "sampler": "SMOTE", "best_params": "{}", "features": ["TXN_vol_20d", "NFCI_ret_5d__div__NFCI_vol_20d", "gjr_condvar_h1", "NFCI_ret_5d", "vix_vs_ma20__zrel__XLB_Materials_zscore_60d", "vix_zscore_10d__zrel__XLB_Materials_zscore_60d", "NWL_Newell_ret_20d__macross__VIX_zscore_60d", "heston_var_ev_h5", "VIX_Price_ret_5d__minus__US2Y_Rate_ret_5d", "Russell_Price_ret_5d__ret5x__HD_zscore_60d", "NFCI_ret_5d__minus__CCI_CrownCastle_vol_20d"], "is_new": false}, {"model_id": "v1_h3_GLOBAL_GradientBoosting_N12", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 3, "n_features": 12, "F1_dir": 0.5484, "F1_UP_FORT": 0.2721, "F1_DOWN_FORT": 0.4005, "train_start": "2001-02-07", "sampler": "SMOTE", "best_params": "{}", "features": ["TXN_vol_20d", "NFCI_ret_5d__div__NFCI_vol_20d", "gjr_condvar_h1", "NFCI_ret_5d", "vix_vs_ma20__zrel__XLB_Materials_zscore_60d", "vix_zscore_10d__zrel__XLB_Materials_zscore_60d", "NWL_Newell_ret_20d__macross__VIX_zscore_60d", "heston_var_ev_h5", "VIX_Price_ret_5d__minus__US2Y_Rate_ret_5d", "Russell_Price_ret_5d__ret5x__HD_zscore_60d", "NFCI_ret_5d__minus__CCI_CrownCastle_vol_20d", "CCI_CrownCastle_vol_20d__macross__heston_kappa"], "is_new": false}, {"model_id": "v1_h3_GLOBAL_GradientBoosting_N13", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 3, "n_features": 13, "F1_dir": 0.5571, "F1_UP_FORT": 0.2347, "F1_DOWN_FORT": 0.4069, "train_start": "2001-02-07", "sampler": "SMOTE", "best_params": "{}", "features": ["TXN_vol_20d", "NFCI_ret_5d__div__NFCI_vol_20d", "gjr_condvar_h1", "NFCI_ret_5d", "vix_vs_ma20__zrel__XLB_Materials_zscore_60d", "vix_zscore_10d__zrel__XLB_Materials_zscore_60d", "NWL_Newell_ret_20d__macross__VIX_zscore_60d", "heston_var_ev_h5", "VIX_Price_ret_5d__minus__US2Y_Rate_ret_5d", "Russell_Price_ret_5d__ret5x__HD_zscore_60d", "NFCI_ret_5d__minus__CCI_CrownCastle_vol_20d", "CCI_CrownCastle_vol_20d__macross__heston_kappa", "vix_vs_ma20__div__NWL_Newell_ret_20d"], "is_new": false}, {"model_id": "v1_h3_GLOBAL_GradientBoosting_N14", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 3, "n_features": 14, "F1_dir": 0.5593, "F1_UP_FORT": 0.2748, "F1_DOWN_FORT": 0.3975, "train_start": "2001-02-07", "sampler": "SMOTE", "best_params": "{}", "features": ["TXN_vol_20d", "NFCI_ret_5d__div__NFCI_vol_20d", "gjr_condvar_h1", "NFCI_ret_5d", "vix_vs_ma20__zrel__XLB_Materials_zscore_60d", "vix_zscore_10d__zrel__XLB_Materials_zscore_60d", "NWL_Newell_ret_20d__macross__VIX_zscore_60d", "heston_var_ev_h5", "VIX_Price_ret_5d__minus__US2Y_Rate_ret_5d", "Russell_Price_ret_5d__ret5x__HD_zscore_60d", "NFCI_ret_5d__minus__CCI_CrownCastle_vol_20d", "CCI_CrownCastle_vol_20d__macross__heston_kappa", "vix_vs_ma20__div__NWL_Newell_ret_20d", "NFCI_vol_20d__ret5x__heston_kappa"], "is_new": false}, {"model_id": "v1_h3_GLOBAL_GradientBoosting_N15", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 3, "n_features": 15, "F1_dir": 0.5608, "F1_UP_FORT": 0.2813, "F1_DOWN_FORT": 0.4086, "train_start": "2001-02-07", "sampler": "SMOTE", "best_params": "{}", "features": ["TXN_vol_20d", "NFCI_ret_5d__div__NFCI_vol_20d", "gjr_condvar_h1", "NFCI_ret_5d", "vix_vs_ma20__zrel__XLB_Materials_zscore_60d", "vix_zscore_10d__zrel__XLB_Materials_zscore_60d", "NWL_Newell_ret_20d__macross__VIX_zscore_60d", "heston_var_ev_h5", "VIX_Price_ret_5d__minus__US2Y_Rate_ret_5d", "Russell_Price_ret_5d__ret5x__HD_zscore_60d", "NFCI_ret_5d__minus__CCI_CrownCastle_vol_20d", "CCI_CrownCastle_vol_20d__macross__heston_kappa", "vix_vs_ma20__div__NWL_Newell_ret_20d", "NFCI_vol_20d__ret5x__heston_kappa", "NFCI_ret_5d__zrel__Russell_Price_ret_5d"], "is_new": false}, {"model_id": "v1_h3_GLOBAL_GradientBoosting_N16", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 3, "n_features": 16, "F1_dir": 0.5691, "F1_UP_FORT": 0.288, "F1_DOWN_FORT": 0.4136, "train_start": "2001-02-07", "sampler": "SMOTE", "best_params": "{}", "features": ["TXN_vol_20d", "NFCI_ret_5d__div__NFCI_vol_20d", "gjr_condvar_h1", "NFCI_ret_5d", "vix_vs_ma20__zrel__XLB_Materials_zscore_60d", "vix_zscore_10d__zrel__XLB_Materials_zscore_60d", "NWL_Newell_ret_20d__macross__VIX_zscore_60d", "heston_var_ev_h5", "VIX_Price_ret_5d__minus__US2Y_Rate_ret_5d", "Russell_Price_ret_5d__ret5x__HD_zscore_60d", "NFCI_ret_5d__minus__CCI_CrownCastle_vol_20d", "CCI_CrownCastle_vol_20d__macross__heston_kappa", "vix_vs_ma20__div__NWL_Newell_ret_20d", "NFCI_vol_20d__ret5x__heston_kappa", "NFCI_ret_5d__zrel__Russell_Price_ret_5d", "NWL_Newell_ret_20d__ret5x__VIX_zscore_60d"], "is_new": false}, {"model_id": "v1_h3_GLOBAL_GradientBoosting_N17", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 3, "n_features": 17, "F1_dir": 0.5576, "F1_UP_FORT": 0.2443, "F1_DOWN_FORT": 0.402, "train_start": "2001-02-07", "sampler": "SMOTE", "best_params": "{}", "features": ["TXN_vol_20d", "NFCI_ret_5d__div__NFCI_vol_20d", "gjr_condvar_h1", "NFCI_ret_5d", "vix_vs_ma20__zrel__XLB_Materials_zscore_60d", "vix_zscore_10d__zrel__XLB_Materials_zscore_60d", "NWL_Newell_ret_20d__macross__VIX_zscore_60d", "heston_var_ev_h5", "VIX_Price_ret_5d__minus__US2Y_Rate_ret_5d", "Russell_Price_ret_5d__ret5x__HD_zscore_60d", "NFCI_ret_5d__minus__CCI_CrownCastle_vol_20d", "CCI_CrownCastle_vol_20d__macross__heston_kappa", "vix_vs_ma20__div__NWL_Newell_ret_20d", "NFCI_vol_20d__ret5x__heston_kappa", "NFCI_ret_5d__zrel__Russell_Price_ret_5d", "NWL_Newell_ret_20d__ret5x__VIX_zscore_60d", "BTI_BritishAmerican_ret_5d"], "is_new": false}, {"model_id": "v1_h3_GLOBAL_GradientBoosting_N18", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 3, "n_features": 18, "F1_dir": 0.5513, "F1_UP_FORT": 0.2782, "F1_DOWN_FORT": 0.4041, "train_start": "2001-02-07", "sampler": "SMOTE", "best_params": "{}", "features": ["TXN_vol_20d", "NFCI_ret_5d__div__NFCI_vol_20d", "gjr_condvar_h1", "NFCI_ret_5d", "vix_vs_ma20__zrel__XLB_Materials_zscore_60d", "vix_zscore_10d__zrel__XLB_Materials_zscore_60d", "NWL_Newell_ret_20d__macross__VIX_zscore_60d", "heston_var_ev_h5", "VIX_Price_ret_5d__minus__US2Y_Rate_ret_5d", "Russell_Price_ret_5d__ret5x__HD_zscore_60d", "NFCI_ret_5d__minus__CCI_CrownCastle_vol_20d", "CCI_CrownCastle_vol_20d__macross__heston_kappa", "vix_vs_ma20__div__NWL_Newell_ret_20d", "NFCI_vol_20d__ret5x__heston_kappa", "NFCI_ret_5d__zrel__Russell_Price_ret_5d", "NWL_Newell_ret_20d__ret5x__VIX_zscore_60d", "BTI_BritishAmerican_ret_5d", "heston_ev_h3"], "is_new": false}, {"model_id": "v1_h3_GLOBAL_RandomForest_N5", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 3, "n_features": 5, "F1_dir": 0.5765, "F1_UP_FORT": 0.2748, "F1_DOWN_FORT": 0.4477, "train_start": "2001-02-07", "sampler": "SMOTE", "best_params": "{}", "features": ["TXN_vol_20d", "NFCI_ret_5d__div__NFCI_vol_20d", "gjr_condvar_h1", "NFCI_ret_5d", "vix_vs_ma20__zrel__XLB_Materials_zscore_60d"], "is_new": false}, {"model_id": "v1_h3_GLOBAL_RandomForest_N6", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 3, "n_features": 6, "F1_dir": 0.5688, "F1_UP_FORT": 0.2749, "F1_DOWN_FORT": 0.4234, "train_start": "2001-02-07", "sampler": "SMOTE", "best_params": "{}", "features": ["TXN_vol_20d", "NFCI_ret_5d__div__NFCI_vol_20d", "gjr_condvar_h1", "NFCI_ret_5d", "vix_vs_ma20__zrel__XLB_Materials_zscore_60d", "vix_zscore_10d__zrel__XLB_Materials_zscore_60d"], "is_new": false}, {"model_id": "v1_h3_GLOBAL_RandomForest_N7", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 3, "n_features": 7, "F1_dir": 0.5637, "F1_UP_FORT": 0.242, "F1_DOWN_FORT": 0.4136, "train_start": "2001-02-07", "sampler": "SMOTE", "best_params": "{}", "features": ["TXN_vol_20d", "NFCI_ret_5d__div__NFCI_vol_20d", "gjr_condvar_h1", "NFCI_ret_5d", "vix_vs_ma20__zrel__XLB_Materials_zscore_60d", "vix_zscore_10d__zrel__XLB_Materials_zscore_60d", "NWL_Newell_ret_20d__macross__VIX_zscore_60d"], "is_new": false}, {"model_id": "v1_h3_GLOBAL_RandomForest_N8", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 3, "n_features": 8, "F1_dir": 0.5616, "F1_UP_FORT": 0.2496, "F1_DOWN_FORT": 0.4429, "train_start": "2001-02-07", "sampler": "SMOTE", "best_params": "{}", "features": ["TXN_vol_20d", "NFCI_ret_5d__div__NFCI_vol_20d", "gjr_condvar_h1", "NFCI_ret_5d", "vix_vs_ma20__zrel__XLB_Materials_zscore_60d", "vix_zscore_10d__zrel__XLB_Materials_zscore_60d", "NWL_Newell_ret_20d__macross__VIX_zscore_60d", "heston_var_ev_h5"], "is_new": false}, {"model_id": "v1_h3_GLOBAL_RandomForest_N9", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 3, "n_features": 9, "F1_dir": 0.5616, "F1_UP_FORT": 0.2578, "F1_DOWN_FORT": 0.4476, "train_start": "2001-02-07", "sampler": "SMOTE", "best_params": "{}", "features": ["TXN_vol_20d", "NFCI_ret_5d__div__NFCI_vol_20d", "gjr_condvar_h1", "NFCI_ret_5d", "vix_vs_ma20__zrel__XLB_Materials_zscore_60d", "vix_zscore_10d__zrel__XLB_Materials_zscore_60d", "NWL_Newell_ret_20d__macross__VIX_zscore_60d", "heston_var_ev_h5", "VIX_Price_ret_5d__minus__US2Y_Rate_ret_5d"], "is_new": false}, {"model_id": "v1_h3_GLOBAL_RandomForest_N10", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 3, "n_features": 10, "F1_dir": 0.5753, "F1_UP_FORT": 0.2763, "F1_DOWN_FORT": 0.4476, "train_start": "2001-02-07", "sampler": "SMOTE", "best_params": "{}", "features": ["TXN_vol_20d", "NFCI_ret_5d__div__NFCI_vol_20d", "gjr_condvar_h1", "NFCI_ret_5d", "vix_vs_ma20__zrel__XLB_Materials_zscore_60d", "vix_zscore_10d__zrel__XLB_Materials_zscore_60d", "NWL_Newell_ret_20d__macross__VIX_zscore_60d", "heston_var_ev_h5", "VIX_Price_ret_5d__minus__US2Y_Rate_ret_5d", "Russell_Price_ret_5d__ret5x__HD_zscore_60d"], "is_new": false}, {"model_id": "v1_h3_GLOBAL_RandomForest_N11", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 3, "n_features": 11, "F1_dir": 0.5718, "F1_UP_FORT": 0.2794, "F1_DOWN_FORT": 0.455, "train_start": "2001-02-07", "sampler": "SMOTE", "best_params": "{}", "features": ["TXN_vol_20d", "NFCI_ret_5d__div__NFCI_vol_20d", "gjr_condvar_h1", "NFCI_ret_5d", "vix_vs_ma20__zrel__XLB_Materials_zscore_60d", "vix_zscore_10d__zrel__XLB_Materials_zscore_60d", "NWL_Newell_ret_20d__macross__VIX_zscore_60d", "heston_var_ev_h5", "VIX_Price_ret_5d__minus__US2Y_Rate_ret_5d", "Russell_Price_ret_5d__ret5x__HD_zscore_60d", "NFCI_ret_5d__minus__CCI_CrownCastle_vol_20d"], "is_new": false}, {"model_id": "v1_h3_GLOBAL_RandomForest_N12", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 3, "n_features": 12, "F1_dir": 0.5742, "F1_UP_FORT": 0.2733, "F1_DOWN_FORT": 0.4316, "train_start": "2001-02-07", "sampler": "SMOTE", "best_params": "{}", "features": ["TXN_vol_20d", "NFCI_ret_5d__div__NFCI_vol_20d", "gjr_condvar_h1", "NFCI_ret_5d", "vix_vs_ma20__zrel__XLB_Materials_zscore_60d", "vix_zscore_10d__zrel__XLB_Materials_zscore_60d", "NWL_Newell_ret_20d__macross__VIX_zscore_60d", "heston_var_ev_h5", "VIX_Price_ret_5d__minus__US2Y_Rate_ret_5d", "Russell_Price_ret_5d__ret5x__HD_zscore_60d", "NFCI_ret_5d__minus__CCI_CrownCastle_vol_20d", "CCI_CrownCastle_vol_20d__macross__heston_kappa"], "is_new": false}, {"model_id": "v1_h3_GLOBAL_RandomForest_N13", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 3, "n_features": 13, "F1_dir": 0.5884, "F1_UP_FORT": 0.3008, "F1_DOWN_FORT": 0.4282, "train_start": "2001-02-07", "sampler": "SMOTE", "best_params": "{}", "features": ["TXN_vol_20d", "NFCI_ret_5d__div__NFCI_vol_20d", "gjr_condvar_h1", "NFCI_ret_5d", "vix_vs_ma20__zrel__XLB_Materials_zscore_60d", "vix_zscore_10d__zrel__XLB_Materials_zscore_60d", "NWL_Newell_ret_20d__macross__VIX_zscore_60d", "heston_var_ev_h5", "VIX_Price_ret_5d__minus__US2Y_Rate_ret_5d", "Russell_Price_ret_5d__ret5x__HD_zscore_60d", "NFCI_ret_5d__minus__CCI_CrownCastle_vol_20d", "CCI_CrownCastle_vol_20d__macross__heston_kappa", "vix_vs_ma20__div__NWL_Newell_ret_20d"], "is_new": false}, {"model_id": "v1_h3_GLOBAL_RandomForest_N14", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 3, "n_features": 14, "F1_dir": 0.5807, "F1_UP_FORT": 0.2843, "F1_DOWN_FORT": 0.4199, "train_start": "2001-02-07", "sampler": "SMOTE", "best_params": "{}", "features": ["TXN_vol_20d", "NFCI_ret_5d__div__NFCI_vol_20d", "gjr_condvar_h1", "NFCI_ret_5d", "vix_vs_ma20__zrel__XLB_Materials_zscore_60d", "vix_zscore_10d__zrel__XLB_Materials_zscore_60d", "NWL_Newell_ret_20d__macross__VIX_zscore_60d", "heston_var_ev_h5", "VIX_Price_ret_5d__minus__US2Y_Rate_ret_5d", "Russell_Price_ret_5d__ret5x__HD_zscore_60d", "NFCI_ret_5d__minus__CCI_CrownCastle_vol_20d", "CCI_CrownCastle_vol_20d__macross__heston_kappa", "vix_vs_ma20__div__NWL_Newell_ret_20d", "NFCI_vol_20d__ret5x__heston_kappa"], "is_new": false}, {"model_id": "v1_h3_GLOBAL_RandomForest_N15", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 3, "n_features": 15, "F1_dir": 0.596, "F1_UP_FORT": 0.2843, "F1_DOWN_FORT": 0.427, "train_start": "2001-02-07", "sampler": "SMOTE", "best_params": "{}", "features": ["TXN_vol_20d", "NFCI_ret_5d__div__NFCI_vol_20d", "gjr_condvar_h1", "NFCI_ret_5d", "vix_vs_ma20__zrel__XLB_Materials_zscore_60d", "vix_zscore_10d__zrel__XLB_Materials_zscore_60d", "NWL_Newell_ret_20d__macross__VIX_zscore_60d", "heston_var_ev_h5", "VIX_Price_ret_5d__minus__US2Y_Rate_ret_5d", "Russell_Price_ret_5d__ret5x__HD_zscore_60d", "NFCI_ret_5d__minus__CCI_CrownCastle_vol_20d", "CCI_CrownCastle_vol_20d__macross__heston_kappa", "vix_vs_ma20__div__NWL_Newell_ret_20d", "NFCI_vol_20d__ret5x__heston_kappa", "NFCI_ret_5d__zrel__Russell_Price_ret_5d"], "is_new": false}, {"model_id": "v1_h3_GLOBAL_RandomForest_N16", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 3, "n_features": 16, "F1_dir": 0.5765, "F1_UP_FORT": 0.2816, "F1_DOWN_FORT": 0.4245, "train_start": "2001-02-07", "sampler": "SMOTE", "best_params": "{}", "features": ["TXN_vol_20d", "NFCI_ret_5d__div__NFCI_vol_20d", "gjr_condvar_h1", "NFCI_ret_5d", "vix_vs_ma20__zrel__XLB_Materials_zscore_60d", "vix_zscore_10d__zrel__XLB_Materials_zscore_60d", "NWL_Newell_ret_20d__macross__VIX_zscore_60d", "heston_var_ev_h5", "VIX_Price_ret_5d__minus__US2Y_Rate_ret_5d", "Russell_Price_ret_5d__ret5x__HD_zscore_60d", "NFCI_ret_5d__minus__CCI_CrownCastle_vol_20d", "CCI_CrownCastle_vol_20d__macross__heston_kappa", "vix_vs_ma20__div__NWL_Newell_ret_20d", "NFCI_vol_20d__ret5x__heston_kappa", "NFCI_ret_5d__zrel__Russell_Price_ret_5d", "NWL_Newell_ret_20d__ret5x__VIX_zscore_60d"], "is_new": false}, {"model_id": "v1_h3_GLOBAL_RandomForest_N17", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 3, "n_features": 17, "F1_dir": 0.5856, "F1_UP_FORT": 0.2791, "F1_DOWN_FORT": 0.433, "train_start": "2001-02-07", "sampler": "SMOTE", "best_params": "{}", "features": ["TXN_vol_20d", "NFCI_ret_5d__div__NFCI_vol_20d", "gjr_condvar_h1", "NFCI_ret_5d", "vix_vs_ma20__zrel__XLB_Materials_zscore_60d", "vix_zscore_10d__zrel__XLB_Materials_zscore_60d", "NWL_Newell_ret_20d__macross__VIX_zscore_60d", "heston_var_ev_h5", "VIX_Price_ret_5d__minus__US2Y_Rate_ret_5d", "Russell_Price_ret_5d__ret5x__HD_zscore_60d", "NFCI_ret_5d__minus__CCI_CrownCastle_vol_20d", "CCI_CrownCastle_vol_20d__macross__heston_kappa", "vix_vs_ma20__div__NWL_Newell_ret_20d", "NFCI_vol_20d__ret5x__heston_kappa", "NFCI_ret_5d__zrel__Russell_Price_ret_5d", "NWL_Newell_ret_20d__ret5x__VIX_zscore_60d", "BTI_BritishAmerican_ret_5d"], "is_new": false}, {"model_id": "v1_h3_GLOBAL_RandomForest_N18", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 3, "n_features": 18, "F1_dir": 0.5765, "F1_UP_FORT": 0.29, "F1_DOWN_FORT": 0.4378, "train_start": "2001-02-07", "sampler": "SMOTE", "best_params": "{}", "features": ["TXN_vol_20d", "NFCI_ret_5d__div__NFCI_vol_20d", "gjr_condvar_h1", "NFCI_ret_5d", "vix_vs_ma20__zrel__XLB_Materials_zscore_60d", "vix_zscore_10d__zrel__XLB_Materials_zscore_60d", "NWL_Newell_ret_20d__macross__VIX_zscore_60d", "heston_var_ev_h5", "VIX_Price_ret_5d__minus__US2Y_Rate_ret_5d", "Russell_Price_ret_5d__ret5x__HD_zscore_60d", "NFCI_ret_5d__minus__CCI_CrownCastle_vol_20d", "CCI_CrownCastle_vol_20d__macross__heston_kappa", "vix_vs_ma20__div__NWL_Newell_ret_20d", "NFCI_vol_20d__ret5x__heston_kappa", "NFCI_ret_5d__zrel__Russell_Price_ret_5d", "NWL_Newell_ret_20d__ret5x__VIX_zscore_60d", "BTI_BritishAmerican_ret_5d", "heston_ev_h3"], "is_new": false}, {"model_id": "v1_h3_GLOBAL_LogisticRegression_N5", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 3, "n_features": 5, "F1_dir": 0.5859, "F1_UP_FORT": 0.3239, "F1_DOWN_FORT": 0.4132, "train_start": "2001-02-07", "sampler": "SMOTE", "best_params": "{}", "features": ["TXN_vol_20d", "NFCI_ret_5d__div__NFCI_vol_20d", "gjr_condvar_h1", "NFCI_ret_5d", "vix_vs_ma20__zrel__XLB_Materials_zscore_60d"], "is_new": false}, {"model_id": "v1_h3_GLOBAL_LogisticRegression_N6", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 3, "n_features": 6, "F1_dir": 0.5837, "F1_UP_FORT": 0.2921, "F1_DOWN_FORT": 0.4185, "train_start": "2001-02-07", "sampler": "SMOTE", "best_params": "{}", "features": ["TXN_vol_20d", "NFCI_ret_5d__div__NFCI_vol_20d", "gjr_condvar_h1", "NFCI_ret_5d", "vix_vs_ma20__zrel__XLB_Materials_zscore_60d", "vix_zscore_10d__zrel__XLB_Materials_zscore_60d"], "is_new": false}, {"model_id": "v1_h3_GLOBAL_LogisticRegression_N7", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 3, "n_features": 7, "F1_dir": 0.5747, "F1_UP_FORT": 0.2786, "F1_DOWN_FORT": 0.4114, "train_start": "2001-02-07", "sampler": "SMOTE", "best_params": "{}", "features": ["TXN_vol_20d", "NFCI_ret_5d__div__NFCI_vol_20d", "gjr_condvar_h1", "NFCI_ret_5d", "vix_vs_ma20__zrel__XLB_Materials_zscore_60d", "vix_zscore_10d__zrel__XLB_Materials_zscore_60d", "NWL_Newell_ret_20d__macross__VIX_zscore_60d"], "is_new": false}, {"model_id": "v1_h3_GLOBAL_LogisticRegression_N8", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 3, "n_features": 8, "F1_dir": 0.5552, "F1_UP_FORT": 0.2682, "F1_DOWN_FORT": 0.4275, "train_start": "2001-02-07", "sampler": "SMOTE", "best_params": "{}", "features": ["TXN_vol_20d", "NFCI_ret_5d__div__NFCI_vol_20d", "gjr_condvar_h1", "NFCI_ret_5d", "vix_vs_ma20__zrel__XLB_Materials_zscore_60d", "vix_zscore_10d__zrel__XLB_Materials_zscore_60d", "NWL_Newell_ret_20d__macross__VIX_zscore_60d", "heston_var_ev_h5"], "is_new": false}, {"model_id": "v1_h3_GLOBAL_LogisticRegression_N9", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 3, "n_features": 9, "F1_dir": 0.5522, "F1_UP_FORT": 0.273, "F1_DOWN_FORT": 0.4308, "train_start": "2001-02-07", "sampler": "SMOTE", "best_params": "{}", "features": ["TXN_vol_20d", "NFCI_ret_5d__div__NFCI_vol_20d", "gjr_condvar_h1", "NFCI_ret_5d", "vix_vs_ma20__zrel__XLB_Materials_zscore_60d", "vix_zscore_10d__zrel__XLB_Materials_zscore_60d", "NWL_Newell_ret_20d__macross__VIX_zscore_60d", "heston_var_ev_h5", "VIX_Price_ret_5d__minus__US2Y_Rate_ret_5d"], "is_new": false}, {"model_id": "v1_h3_GLOBAL_LogisticRegression_N10", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 3, "n_features": 10, "F1_dir": 0.5454, "F1_UP_FORT": 0.2748, "F1_DOWN_FORT": 0.4287, "train_start": "2001-02-07", "sampler": "SMOTE", "best_params": "{}", "features": ["TXN_vol_20d", "NFCI_ret_5d__div__NFCI_vol_20d", "gjr_condvar_h1", "NFCI_ret_5d", "vix_vs_ma20__zrel__XLB_Materials_zscore_60d", "vix_zscore_10d__zrel__XLB_Materials_zscore_60d", "NWL_Newell_ret_20d__macross__VIX_zscore_60d", "heston_var_ev_h5", "VIX_Price_ret_5d__minus__US2Y_Rate_ret_5d", "Russell_Price_ret_5d__ret5x__HD_zscore_60d"], "is_new": false}, {"model_id": "v1_h3_GLOBAL_LogisticRegression_N11", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 3, "n_features": 11, "F1_dir": 0.545, "F1_UP_FORT": 0.2694, "F1_DOWN_FORT": 0.4334, "train_start": "2001-02-07", "sampler": "SMOTE", "best_params": "{}", "features": ["TXN_vol_20d", "NFCI_ret_5d__div__NFCI_vol_20d", "gjr_condvar_h1", "NFCI_ret_5d", "vix_vs_ma20__zrel__XLB_Materials_zscore_60d", "vix_zscore_10d__zrel__XLB_Materials_zscore_60d", "NWL_Newell_ret_20d__macross__VIX_zscore_60d", "heston_var_ev_h5", "VIX_Price_ret_5d__minus__US2Y_Rate_ret_5d", "Russell_Price_ret_5d__ret5x__HD_zscore_60d", "NFCI_ret_5d__minus__CCI_CrownCastle_vol_20d"], "is_new": false}, {"model_id": "v1_h3_GLOBAL_LogisticRegression_N12", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 3, "n_features": 12, "F1_dir": 0.5457, "F1_UP_FORT": 0.2852, "F1_DOWN_FORT": 0.4213, "train_start": "2001-02-07", "sampler": "SMOTE", "best_params": "{}", "features": ["TXN_vol_20d", "NFCI_ret_5d__div__NFCI_vol_20d", "gjr_condvar_h1", "NFCI_ret_5d", "vix_vs_ma20__zrel__XLB_Materials_zscore_60d", "vix_zscore_10d__zrel__XLB_Materials_zscore_60d", "NWL_Newell_ret_20d__macross__VIX_zscore_60d", "heston_var_ev_h5", "VIX_Price_ret_5d__minus__US2Y_Rate_ret_5d", "Russell_Price_ret_5d__ret5x__HD_zscore_60d", "NFCI_ret_5d__minus__CCI_CrownCastle_vol_20d", "CCI_CrownCastle_vol_20d__macross__heston_kappa"], "is_new": false}, {"model_id": "v1_h3_GLOBAL_LogisticRegression_N13", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 3, "n_features": 13, "F1_dir": 0.5502, "F1_UP_FORT": 0.2738, "F1_DOWN_FORT": 0.4283, "train_start": "2001-02-07", "sampler": "SMOTE", "best_params": "{}", "features": ["TXN_vol_20d", "NFCI_ret_5d__div__NFCI_vol_20d", "gjr_condvar_h1", "NFCI_ret_5d", "vix_vs_ma20__zrel__XLB_Materials_zscore_60d", "vix_zscore_10d__zrel__XLB_Materials_zscore_60d", "NWL_Newell_ret_20d__macross__VIX_zscore_60d", "heston_var_ev_h5", "VIX_Price_ret_5d__minus__US2Y_Rate_ret_5d", "Russell_Price_ret_5d__ret5x__HD_zscore_60d", "NFCI_ret_5d__minus__CCI_CrownCastle_vol_20d", "CCI_CrownCastle_vol_20d__macross__heston_kappa", "vix_vs_ma20__div__NWL_Newell_ret_20d"], "is_new": false}, {"model_id": "v1_h3_GLOBAL_LogisticRegression_N14", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 3, "n_features": 14, "F1_dir": 0.553, "F1_UP_FORT": 0.2852, "F1_DOWN_FORT": 0.4174, "train_start": "2001-02-07", "sampler": "SMOTE", "best_params": "{}", "features": ["TXN_vol_20d", "NFCI_ret_5d__div__NFCI_vol_20d", "gjr_condvar_h1", "NFCI_ret_5d", "vix_vs_ma20__zrel__XLB_Materials_zscore_60d", "vix_zscore_10d__zrel__XLB_Materials_zscore_60d", "NWL_Newell_ret_20d__macross__VIX_zscore_60d", "heston_var_ev_h5", "VIX_Price_ret_5d__minus__US2Y_Rate_ret_5d", "Russell_Price_ret_5d__ret5x__HD_zscore_60d", "NFCI_ret_5d__minus__CCI_CrownCastle_vol_20d", "CCI_CrownCastle_vol_20d__macross__heston_kappa", "vix_vs_ma20__div__NWL_Newell_ret_20d", "NFCI_vol_20d__ret5x__heston_kappa"], "is_new": false}, {"model_id": "v1_h3_GLOBAL_LogisticRegression_N15", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 3, "n_features": 15, "F1_dir": 0.5534, "F1_UP_FORT": 0.2857, "F1_DOWN_FORT": 0.4248, "train_start": "2001-02-07", "sampler": "SMOTE", "best_params": "{}", "features": ["TXN_vol_20d", "NFCI_ret_5d__div__NFCI_vol_20d", "gjr_condvar_h1", "NFCI_ret_5d", "vix_vs_ma20__zrel__XLB_Materials_zscore_60d", "vix_zscore_10d__zrel__XLB_Materials_zscore_60d", "NWL_Newell_ret_20d__macross__VIX_zscore_60d", "heston_var_ev_h5", "VIX_Price_ret_5d__minus__US2Y_Rate_ret_5d", "Russell_Price_ret_5d__ret5x__HD_zscore_60d", "NFCI_ret_5d__minus__CCI_CrownCastle_vol_20d", "CCI_CrownCastle_vol_20d__macross__heston_kappa", "vix_vs_ma20__div__NWL_Newell_ret_20d", "NFCI_vol_20d__ret5x__heston_kappa", "NFCI_ret_5d__zrel__Russell_Price_ret_5d"], "is_new": false}, {"model_id": "v1_h3_GLOBAL_LogisticRegression_N16", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 3, "n_features": 16, "F1_dir": 0.5565, "F1_UP_FORT": 0.276, "F1_DOWN_FORT": 0.4231, "train_start": "2001-02-07", "sampler": "SMOTE", "best_params": "{}", "features": ["TXN_vol_20d", "NFCI_ret_5d__div__NFCI_vol_20d", "gjr_condvar_h1", "NFCI_ret_5d", "vix_vs_ma20__zrel__XLB_Materials_zscore_60d", "vix_zscore_10d__zrel__XLB_Materials_zscore_60d", "NWL_Newell_ret_20d__macross__VIX_zscore_60d", "heston_var_ev_h5", "VIX_Price_ret_5d__minus__US2Y_Rate_ret_5d", "Russell_Price_ret_5d__ret5x__HD_zscore_60d", "NFCI_ret_5d__minus__CCI_CrownCastle_vol_20d", "CCI_CrownCastle_vol_20d__macross__heston_kappa", "vix_vs_ma20__div__NWL_Newell_ret_20d", "NFCI_vol_20d__ret5x__heston_kappa", "NFCI_ret_5d__zrel__Russell_Price_ret_5d", "NWL_Newell_ret_20d__ret5x__VIX_zscore_60d"], "is_new": false}, {"model_id": "v1_h3_GLOBAL_LogisticRegression_N17", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 3, "n_features": 17, "F1_dir": 0.5468, "F1_UP_FORT": 0.283, "F1_DOWN_FORT": 0.4197, "train_start": "2001-02-07", "sampler": "SMOTE", "best_params": "{}", "features": ["TXN_vol_20d", "NFCI_ret_5d__div__NFCI_vol_20d", "gjr_condvar_h1", "NFCI_ret_5d", "vix_vs_ma20__zrel__XLB_Materials_zscore_60d", "vix_zscore_10d__zrel__XLB_Materials_zscore_60d", "NWL_Newell_ret_20d__macross__VIX_zscore_60d", "heston_var_ev_h5", "VIX_Price_ret_5d__minus__US2Y_Rate_ret_5d", "Russell_Price_ret_5d__ret5x__HD_zscore_60d", "NFCI_ret_5d__minus__CCI_CrownCastle_vol_20d", "CCI_CrownCastle_vol_20d__macross__heston_kappa", "vix_vs_ma20__div__NWL_Newell_ret_20d", "NFCI_vol_20d__ret5x__heston_kappa", "NFCI_ret_5d__zrel__Russell_Price_ret_5d", "NWL_Newell_ret_20d__ret5x__VIX_zscore_60d", "BTI_BritishAmerican_ret_5d"], "is_new": false}, {"model_id": "v1_h3_GLOBAL_LogisticRegression_N18", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 3, "n_features": 18, "F1_dir": 0.5508, "F1_UP_FORT": 0.2911, "F1_DOWN_FORT": 0.4119, "train_start": "2001-02-07", "sampler": "SMOTE", "best_params": "{}", "features": ["TXN_vol_20d", "NFCI_ret_5d__div__NFCI_vol_20d", "gjr_condvar_h1", "NFCI_ret_5d", "vix_vs_ma20__zrel__XLB_Materials_zscore_60d", "vix_zscore_10d__zrel__XLB_Materials_zscore_60d", "NWL_Newell_ret_20d__macross__VIX_zscore_60d", "heston_var_ev_h5", "VIX_Price_ret_5d__minus__US2Y_Rate_ret_5d", "Russell_Price_ret_5d__ret5x__HD_zscore_60d", "NFCI_ret_5d__minus__CCI_CrownCastle_vol_20d", "CCI_CrownCastle_vol_20d__macross__heston_kappa", "vix_vs_ma20__div__NWL_Newell_ret_20d", "NFCI_vol_20d__ret5x__heston_kappa", "NFCI_ret_5d__zrel__Russell_Price_ret_5d", "NWL_Newell_ret_20d__ret5x__VIX_zscore_60d", "BTI_BritishAmerican_ret_5d", "heston_ev_h3"], "is_new": false}, {"model_id": "v1_h3_GLOBAL_XGBoost_Optuna_N8", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 3, "n_features": 8, "F1_dir": 0.5488, "F1_UP_FORT": 0.2278, "F1_DOWN_FORT": 0.4095, "train_start": "2001-02-07", "sampler": "SMOTE", "best_params": "{}", "features": ["TXN_vol_20d", "NFCI_ret_5d__div__NFCI_vol_20d", "gjr_condvar_h1", "NFCI_ret_5d", "vix_vs_ma20__zrel__XLB_Materials_zscore_60d", "vix_zscore_10d__zrel__XLB_Materials_zscore_60d", "NWL_Newell_ret_20d__macross__VIX_zscore_60d", "heston_var_ev_h5"], "is_new": false}, {"model_id": "v1_h3_GLOBAL_XGBoost_OptunaCal_N8", "algo": "XGBoostCal", "regime": "GLOBAL", "horizon": 3, "n_features": 8, "F1_dir": 0.5408, "F1_UP_FORT": 0.2272, "F1_DOWN_FORT": 0.3429, "train_start": "2001-02-07", "sampler": "SMOTE", "best_params": "{}", "features": ["TXN_vol_20d", "NFCI_ret_5d__div__NFCI_vol_20d", "gjr_condvar_h1", "NFCI_ret_5d", "vix_vs_ma20__zrel__XLB_Materials_zscore_60d", "vix_zscore_10d__zrel__XLB_Materials_zscore_60d", "NWL_Newell_ret_20d__macross__VIX_zscore_60d", "heston_var_ev_h5"], "is_new": false}, {"model_id": "v1_h5_CALM_XGBoost_N6", "algo": "XGBoost", "regime": "CALM", "horizon": 5, "n_features": 6, "F1_dir": 0.5081, "F1_UP_FORT": 0.1463, "F1_DOWN_FORT": 0.378, "train_start": "2000-11-16", "sampler": "SMOTETomek", "best_params": "{}", "features": ["XLY_Disc_vol_20d", "CAT_Caterpillar_ret_5d__macross__VRTX_VertexPharm_zscore_60d", "NFCI_ret_5d", "EWA_Australia_zscore_60d", "BAX_BankBoston_vol_20d__div__VRTX_VertexPharm_zscore_60d", "MSTR_Bitcoin3_ret_1d"], "is_new": false}, {"model_id": "v1_h5_CALM_XGBoost_N7", "algo": "XGBoost", "regime": "CALM", "horizon": 5, "n_features": 7, "F1_dir": 0.5513, "F1_UP_FORT": 0.15, "F1_DOWN_FORT": 0.4, "train_start": "2000-11-16", "sampler": "SMOTETomek", "best_params": "{}", "features": ["XLY_Disc_vol_20d", "CAT_Caterpillar_ret_5d__macross__VRTX_VertexPharm_zscore_60d", "NFCI_ret_5d", "EWA_Australia_zscore_60d", "BAX_BankBoston_vol_20d__div__VRTX_VertexPharm_zscore_60d", "MSTR_Bitcoin3_ret_1d", "vix_vs_ma10__div__vix_zscore_10d"], "is_new": false}, {"model_id": "v1_h5_CALM_XGBoost_N8", "algo": "XGBoost", "regime": "CALM", "horizon": 5, "n_features": 8, "F1_dir": 0.5342, "F1_UP_FORT": 0.175, "F1_DOWN_FORT": 0.303, "train_start": "2000-11-16", "sampler": "SMOTETomek", "best_params": "{}", "features": ["XLY_Disc_vol_20d", "CAT_Caterpillar_ret_5d__macross__VRTX_VertexPharm_zscore_60d", "NFCI_ret_5d", "EWA_Australia_zscore_60d", "BAX_BankBoston_vol_20d__div__VRTX_VertexPharm_zscore_60d", "MSTR_Bitcoin3_ret_1d", "vix_vs_ma10__div__vix_zscore_10d", "AMT_AmericanTower_ret_1d"], "is_new": false}, {"model_id": "v1_h5_CALM_XGBoost_N9", "algo": "XGBoost", "regime": "CALM", "horizon": 5, "n_features": 9, "F1_dir": 0.554, "F1_UP_FORT": 0.1647, "F1_DOWN_FORT": 0.3967, "train_start": "2000-11-16", "sampler": "SMOTETomek", "best_params": "{}", "features": ["XLY_Disc_vol_20d", "CAT_Caterpillar_ret_5d__macross__VRTX_VertexPharm_zscore_60d", "NFCI_ret_5d", "EWA_Australia_zscore_60d", "BAX_BankBoston_vol_20d__div__VRTX_VertexPharm_zscore_60d", "MSTR_Bitcoin3_ret_1d", "vix_vs_ma10__div__vix_zscore_10d", "AMT_AmericanTower_ret_1d", "Brent_Oil_FRED_ret_5d"], "is_new": false}, {"model_id": "v1_h5_CALM_XGBoost_N10", "algo": "XGBoost", "regime": "CALM", "horizon": 5, "n_features": 10, "F1_dir": 0.5491, "F1_UP_FORT": 0.1522, "F1_DOWN_FORT": 0.4, "train_start": "2000-11-16", "sampler": "SMOTETomek", "best_params": "{}", "features": ["XLY_Disc_vol_20d", "CAT_Caterpillar_ret_5d__macross__VRTX_VertexPharm_zscore_60d", "NFCI_ret_5d", "EWA_Australia_zscore_60d", "BAX_BankBoston_vol_20d__div__VRTX_VertexPharm_zscore_60d", "MSTR_Bitcoin3_ret_1d", "vix_vs_ma10__div__vix_zscore_10d", "AMT_AmericanTower_ret_1d", "Brent_Oil_FRED_ret_5d", "vix_zscore_10d__zrel__AMZN_zscore_60d"], "is_new": false}, {"model_id": "v1_h5_CALM_XGBoost_N11", "algo": "XGBoost", "regime": "CALM", "horizon": 5, "n_features": 11, "F1_dir": 0.5666, "F1_UP_FORT": 0.1758, "F1_DOWN_FORT": 0.4138, "train_start": "2000-11-16", "sampler": "SMOTETomek", "best_params": "{}", "features": ["XLY_Disc_vol_20d", "CAT_Caterpillar_ret_5d__macross__VRTX_VertexPharm_zscore_60d", "NFCI_ret_5d", "EWA_Australia_zscore_60d", "BAX_BankBoston_vol_20d__div__VRTX_VertexPharm_zscore_60d", "MSTR_Bitcoin3_ret_1d", "vix_vs_ma10__div__vix_zscore_10d", "AMT_AmericanTower_ret_1d", "Brent_Oil_FRED_ret_5d", "vix_zscore_10d__zrel__AMZN_zscore_60d", "vix_zscore_10d__div__spx_abs_ret_max_5d"], "is_new": false}, {"model_id": "v1_h5_CALM_XGBoost_N12", "algo": "XGBoost", "regime": "CALM", "horizon": 5, "n_features": 12, "F1_dir": 0.5846, "F1_UP_FORT": 0.198, "F1_DOWN_FORT": 0.4144, "train_start": "2000-11-16", "sampler": "SMOTETomek", "best_params": "{}", "features": ["XLY_Disc_vol_20d", "CAT_Caterpillar_ret_5d__macross__VRTX_VertexPharm_zscore_60d", "NFCI_ret_5d", "EWA_Australia_zscore_60d", "BAX_BankBoston_vol_20d__div__VRTX_VertexPharm_zscore_60d", "MSTR_Bitcoin3_ret_1d", "vix_vs_ma10__div__vix_zscore_10d", "AMT_AmericanTower_ret_1d", "Brent_Oil_FRED_ret_5d", "vix_zscore_10d__zrel__AMZN_zscore_60d", "vix_zscore_10d__div__spx_abs_ret_max_5d", "NFCI_ret_5d__div__CPB_CampbellSoup_vol_20d"], "is_new": false}, {"model_id": "v1_h5_CALM_XGBoost_N13", "algo": "XGBoost", "regime": "CALM", "horizon": 5, "n_features": 13, "F1_dir": 0.563, "F1_UP_FORT": 0.2, "F1_DOWN_FORT": 0.403, "train_start": "2000-11-16", "sampler": "SMOTETomek", "best_params": "{}", "features": ["XLY_Disc_vol_20d", "CAT_Caterpillar_ret_5d__macross__VRTX_VertexPharm_zscore_60d", "NFCI_ret_5d", "EWA_Australia_zscore_60d", "BAX_BankBoston_vol_20d__div__VRTX_VertexPharm_zscore_60d", "MSTR_Bitcoin3_ret_1d", "vix_vs_ma10__div__vix_zscore_10d", "AMT_AmericanTower_ret_1d", "Brent_Oil_FRED_ret_5d", "vix_zscore_10d__zrel__AMZN_zscore_60d", "vix_zscore_10d__div__spx_abs_ret_max_5d", "NFCI_ret_5d__div__CPB_CampbellSoup_vol_20d", "CPB_CampbellSoup_zscore_60d"], "is_new": false}, {"model_id": "v1_h5_CALM_XGBoost_N14", "algo": "XGBoost", "regime": "CALM", "horizon": 5, "n_features": 14, "F1_dir": 0.5403, "F1_UP_FORT": 0.1739, "F1_DOWN_FORT": 0.3664, "train_start": "2000-11-16", "sampler": "SMOTETomek", "best_params": "{}", "features": ["XLY_Disc_vol_20d", "CAT_Caterpillar_ret_5d__macross__VRTX_VertexPharm_zscore_60d", "NFCI_ret_5d", "EWA_Australia_zscore_60d", "BAX_BankBoston_vol_20d__div__VRTX_VertexPharm_zscore_60d", "MSTR_Bitcoin3_ret_1d", "vix_vs_ma10__div__vix_zscore_10d", "AMT_AmericanTower_ret_1d", "Brent_Oil_FRED_ret_5d", "vix_zscore_10d__zrel__AMZN_zscore_60d", "vix_zscore_10d__div__spx_abs_ret_max_5d", "NFCI_ret_5d__div__CPB_CampbellSoup_vol_20d", "CPB_CampbellSoup_zscore_60d", "SLB_Schlumberger_ret_1d"], "is_new": false}, {"model_id": "v1_h5_CALM_XGBoost_N15", "algo": "XGBoost", "regime": "CALM", "horizon": 5, "n_features": 15, "F1_dir": 0.6017, "F1_UP_FORT": 0.2247, "F1_DOWN_FORT": 0.4056, "train_start": "2000-11-16", "sampler": "SMOTETomek", "best_params": "{}", "features": ["XLY_Disc_vol_20d", "CAT_Caterpillar_ret_5d__macross__VRTX_VertexPharm_zscore_60d", "NFCI_ret_5d", "EWA_Australia_zscore_60d", "BAX_BankBoston_vol_20d__div__VRTX_VertexPharm_zscore_60d", "MSTR_Bitcoin3_ret_1d", "vix_vs_ma10__div__vix_zscore_10d", "AMT_AmericanTower_ret_1d", "Brent_Oil_FRED_ret_5d", "vix_zscore_10d__zrel__AMZN_zscore_60d", "vix_zscore_10d__div__spx_abs_ret_max_5d", "NFCI_ret_5d__div__CPB_CampbellSoup_vol_20d", "CPB_CampbellSoup_zscore_60d", "SLB_Schlumberger_ret_1d", "HD_ret_1d"], "is_new": false}, {"model_id": "v1_h5_CALM_XGBoost_N16", "algo": "XGBoost", "regime": "CALM", "horizon": 5, "n_features": 16, "F1_dir": 0.5741, "F1_UP_FORT": 0.1915, "F1_DOWN_FORT": 0.378, "train_start": "2000-11-16", "sampler": "SMOTETomek", "best_params": "{}", "features": ["XLY_Disc_vol_20d", "CAT_Caterpillar_ret_5d__macross__VRTX_VertexPharm_zscore_60d", "NFCI_ret_5d", "EWA_Australia_zscore_60d", "BAX_BankBoston_vol_20d__div__VRTX_VertexPharm_zscore_60d", "MSTR_Bitcoin3_ret_1d", "vix_vs_ma10__div__vix_zscore_10d", "AMT_AmericanTower_ret_1d", "Brent_Oil_FRED_ret_5d", "vix_zscore_10d__zrel__AMZN_zscore_60d", "vix_zscore_10d__div__spx_abs_ret_max_5d", "NFCI_ret_5d__div__CPB_CampbellSoup_vol_20d", "CPB_CampbellSoup_zscore_60d", "SLB_Schlumberger_ret_1d", "HD_ret_1d", "vix_vs_ma10__minus__CAT_Caterpillar_ret_5d"], "is_new": false}, {"model_id": "v1_h5_CALM_XGBoost_N17", "algo": "XGBoost", "regime": "CALM", "horizon": 5, "n_features": 17, "F1_dir": 0.5747, "F1_UP_FORT": 0.1875, "F1_DOWN_FORT": 0.375, "train_start": "2000-11-16", "sampler": "SMOTETomek", "best_params": "{}", "features": ["XLY_Disc_vol_20d", "CAT_Caterpillar_ret_5d__macross__VRTX_VertexPharm_zscore_60d", "NFCI_ret_5d", "EWA_Australia_zscore_60d", "BAX_BankBoston_vol_20d__div__VRTX_VertexPharm_zscore_60d", "MSTR_Bitcoin3_ret_1d", "vix_vs_ma10__div__vix_zscore_10d", "AMT_AmericanTower_ret_1d", "Brent_Oil_FRED_ret_5d", "vix_zscore_10d__zrel__AMZN_zscore_60d", "vix_zscore_10d__div__spx_abs_ret_max_5d", "NFCI_ret_5d__div__CPB_CampbellSoup_vol_20d", "CPB_CampbellSoup_zscore_60d", "SLB_Schlumberger_ret_1d", "HD_ret_1d", "vix_vs_ma10__minus__CAT_Caterpillar_ret_5d", "vix_acceleration_1d"], "is_new": false}, {"model_id": "v1_h5_CALM_LightGBM_N6", "algo": "LightGBM", "regime": "CALM", "horizon": 5, "n_features": 6, "F1_dir": 0.5298, "F1_UP_FORT": 0.1928, "F1_DOWN_FORT": 0.3529, "train_start": "2000-11-16", "sampler": "SMOTETomek", "best_params": "{}", "features": ["XLY_Disc_vol_20d", "CAT_Caterpillar_ret_5d__macross__VRTX_VertexPharm_zscore_60d", "NFCI_ret_5d", "EWA_Australia_zscore_60d", "BAX_BankBoston_vol_20d__div__VRTX_VertexPharm_zscore_60d", "MSTR_Bitcoin3_ret_1d"], "is_new": false}, {"model_id": "v1_h5_CALM_LightGBM_N7", "algo": "LightGBM", "regime": "CALM", "horizon": 5, "n_features": 7, "F1_dir": 0.5039, "F1_UP_FORT": 0.1728, "F1_DOWN_FORT": 0.3361, "train_start": "2000-11-16", "sampler": "SMOTETomek", "best_params": "{}", "features": ["XLY_Disc_vol_20d", "CAT_Caterpillar_ret_5d__macross__VRTX_VertexPharm_zscore_60d", "NFCI_ret_5d", "EWA_Australia_zscore_60d", "BAX_BankBoston_vol_20d__div__VRTX_VertexPharm_zscore_60d", "MSTR_Bitcoin3_ret_1d", "vix_vs_ma10__div__vix_zscore_10d"], "is_new": false}, {"model_id": "v1_h5_CALM_LightGBM_N8", "algo": "LightGBM", "regime": "CALM", "horizon": 5, "n_features": 8, "F1_dir": 0.5078, "F1_UP_FORT": 0.2444, "F1_DOWN_FORT": 0.304, "train_start": "2000-11-16", "sampler": "SMOTETomek", "best_params": "{}", "features": ["XLY_Disc_vol_20d", "CAT_Caterpillar_ret_5d__macross__VRTX_VertexPharm_zscore_60d", "NFCI_ret_5d", "EWA_Australia_zscore_60d", "BAX_BankBoston_vol_20d__div__VRTX_VertexPharm_zscore_60d", "MSTR_Bitcoin3_ret_1d", "vix_vs_ma10__div__vix_zscore_10d", "AMT_AmericanTower_ret_1d"], "is_new": false}, {"model_id": "v1_h5_CALM_LightGBM_N9", "algo": "LightGBM", "regime": "CALM", "horizon": 5, "n_features": 9, "F1_dir": 0.5252, "F1_UP_FORT": 0.122, "F1_DOWN_FORT": 0.359, "train_start": "2000-11-16", "sampler": "SMOTETomek", "best_params": "{}", "features": ["XLY_Disc_vol_20d", "CAT_Caterpillar_ret_5d__macross__VRTX_VertexPharm_zscore_60d", "NFCI_ret_5d", "EWA_Australia_zscore_60d", "BAX_BankBoston_vol_20d__div__VRTX_VertexPharm_zscore_60d", "MSTR_Bitcoin3_ret_1d", "vix_vs_ma10__div__vix_zscore_10d", "AMT_AmericanTower_ret_1d", "Brent_Oil_FRED_ret_5d"], "is_new": false}, {"model_id": "v1_h5_CALM_LightGBM_N10", "algo": "LightGBM", "regime": "CALM", "horizon": 5, "n_features": 10, "F1_dir": 0.5703, "F1_UP_FORT": 0.2418, "F1_DOWN_FORT": 0.3495, "train_start": "2000-11-16", "sampler": "SMOTETomek", "best_params": "{}", "features": ["XLY_Disc_vol_20d", "CAT_Caterpillar_ret_5d__macross__VRTX_VertexPharm_zscore_60d", "NFCI_ret_5d", "EWA_Australia_zscore_60d", "BAX_BankBoston_vol_20d__div__VRTX_VertexPharm_zscore_60d", "MSTR_Bitcoin3_ret_1d", "vix_vs_ma10__div__vix_zscore_10d", "AMT_AmericanTower_ret_1d", "Brent_Oil_FRED_ret_5d", "vix_zscore_10d__zrel__AMZN_zscore_60d"], "is_new": false}, {"model_id": "v1_h5_CALM_LightGBM_N11", "algo": "LightGBM", "regime": "CALM", "horizon": 5, "n_features": 11, "F1_dir": 0.558, "F1_UP_FORT": 0.1667, "F1_DOWN_FORT": 0.4034, "train_start": "2000-11-16", "sampler": "SMOTETomek", "best_params": "{}", "features": ["XLY_Disc_vol_20d", "CAT_Caterpillar_ret_5d__macross__VRTX_VertexPharm_zscore_60d", "NFCI_ret_5d", "EWA_Australia_zscore_60d", "BAX_BankBoston_vol_20d__div__VRTX_VertexPharm_zscore_60d", "MSTR_Bitcoin3_ret_1d", "vix_vs_ma10__div__vix_zscore_10d", "AMT_AmericanTower_ret_1d", "Brent_Oil_FRED_ret_5d", "vix_zscore_10d__zrel__AMZN_zscore_60d", "vix_zscore_10d__div__spx_abs_ret_max_5d"], "is_new": false}, {"model_id": "v1_h5_CALM_LightGBM_N12", "algo": "LightGBM", "regime": "CALM", "horizon": 5, "n_features": 12, "F1_dir": 0.5638, "F1_UP_FORT": 0.1905, "F1_DOWN_FORT": 0.458, "train_start": "2000-11-16", "sampler": "SMOTETomek", "best_params": "{}", "features": ["XLY_Disc_vol_20d", "CAT_Caterpillar_ret_5d__macross__VRTX_VertexPharm_zscore_60d", "NFCI_ret_5d", "EWA_Australia_zscore_60d", "BAX_BankBoston_vol_20d__div__VRTX_VertexPharm_zscore_60d", "MSTR_Bitcoin3_ret_1d", "vix_vs_ma10__div__vix_zscore_10d", "AMT_AmericanTower_ret_1d", "Brent_Oil_FRED_ret_5d", "vix_zscore_10d__zrel__AMZN_zscore_60d", "vix_zscore_10d__div__spx_abs_ret_max_5d", "NFCI_ret_5d__div__CPB_CampbellSoup_vol_20d"], "is_new": false}, {"model_id": "v1_h5_CALM_LightGBM_N13", "algo": "LightGBM", "regime": "CALM", "horizon": 5, "n_features": 13, "F1_dir": 0.5547, "F1_UP_FORT": 0.1556, "F1_DOWN_FORT": 0.3937, "train_start": "2000-11-16", "sampler": "SMOTETomek", "best_params": "{}", "features": ["XLY_Disc_vol_20d", "CAT_Caterpillar_ret_5d__macross__VRTX_VertexPharm_zscore_60d", "NFCI_ret_5d", "EWA_Australia_zscore_60d", "BAX_BankBoston_vol_20d__div__VRTX_VertexPharm_zscore_60d", "MSTR_Bitcoin3_ret_1d", "vix_vs_ma10__div__vix_zscore_10d", "AMT_AmericanTower_ret_1d", "Brent_Oil_FRED_ret_5d", "vix_zscore_10d__zrel__AMZN_zscore_60d", "vix_zscore_10d__div__spx_abs_ret_max_5d", "NFCI_ret_5d__div__CPB_CampbellSoup_vol_20d", "CPB_CampbellSoup_zscore_60d"], "is_new": false}, {"model_id": "v1_h5_CALM_LightGBM_N14", "algo": "LightGBM", "regime": "CALM", "horizon": 5, "n_features": 14, "F1_dir": 0.6009, "F1_UP_FORT": 0.1667, "F1_DOWN_FORT": 0.4122, "train_start": "2000-11-16", "sampler": "SMOTETomek", "best_params": "{}", "features": ["XLY_Disc_vol_20d", "CAT_Caterpillar_ret_5d__macross__VRTX_VertexPharm_zscore_60d", "NFCI_ret_5d", "EWA_Australia_zscore_60d", "BAX_BankBoston_vol_20d__div__VRTX_VertexPharm_zscore_60d", "MSTR_Bitcoin3_ret_1d", "vix_vs_ma10__div__vix_zscore_10d", "AMT_AmericanTower_ret_1d", "Brent_Oil_FRED_ret_5d", "vix_zscore_10d__zrel__AMZN_zscore_60d", "vix_zscore_10d__div__spx_abs_ret_max_5d", "NFCI_ret_5d__div__CPB_CampbellSoup_vol_20d", "CPB_CampbellSoup_zscore_60d", "SLB_Schlumberger_ret_1d"], "is_new": false}, {"model_id": "v1_h5_CALM_LightGBM_N15", "algo": "LightGBM", "regime": "CALM", "horizon": 5, "n_features": 15, "F1_dir": 0.5417, "F1_UP_FORT": 0.0759, "F1_DOWN_FORT": 0.3407, "train_start": "2000-11-16", "sampler": "SMOTETomek", "best_params": "{}", "features": ["XLY_Disc_vol_20d", "CAT_Caterpillar_ret_5d__macross__VRTX_VertexPharm_zscore_60d", "NFCI_ret_5d", "EWA_Australia_zscore_60d", "BAX_BankBoston_vol_20d__div__VRTX_VertexPharm_zscore_60d", "MSTR_Bitcoin3_ret_1d", "vix_vs_ma10__div__vix_zscore_10d", "AMT_AmericanTower_ret_1d", "Brent_Oil_FRED_ret_5d", "vix_zscore_10d__zrel__AMZN_zscore_60d", "vix_zscore_10d__div__spx_abs_ret_max_5d", "NFCI_ret_5d__div__CPB_CampbellSoup_vol_20d", "CPB_CampbellSoup_zscore_60d", "SLB_Schlumberger_ret_1d", "HD_ret_1d"], "is_new": false}, {"model_id": "v1_h5_CALM_LightGBM_N16", "algo": "LightGBM", "regime": "CALM", "horizon": 5, "n_features": 16, "F1_dir": 0.6232, "F1_UP_FORT": 0.1556, "F1_DOWN_FORT": 0.4748, "train_start": "2000-11-16", "sampler": "SMOTETomek", "best_params": "{}", "features": ["XLY_Disc_vol_20d", "CAT_Caterpillar_ret_5d__macross__VRTX_VertexPharm_zscore_60d", "NFCI_ret_5d", "EWA_Australia_zscore_60d", "BAX_BankBoston_vol_20d__div__VRTX_VertexPharm_zscore_60d", "MSTR_Bitcoin3_ret_1d", "vix_vs_ma10__div__vix_zscore_10d", "AMT_AmericanTower_ret_1d", "Brent_Oil_FRED_ret_5d", "vix_zscore_10d__zrel__AMZN_zscore_60d", "vix_zscore_10d__div__spx_abs_ret_max_5d", "NFCI_ret_5d__div__CPB_CampbellSoup_vol_20d", "CPB_CampbellSoup_zscore_60d", "SLB_Schlumberger_ret_1d", "HD_ret_1d", "vix_vs_ma10__minus__CAT_Caterpillar_ret_5d"], "is_new": false}, {"model_id": "v1_h5_CALM_LightGBM_N17", "algo": "LightGBM", "regime": "CALM", "horizon": 5, "n_features": 17, "F1_dir": 0.6033, "F1_UP_FORT": 0.172, "F1_DOWN_FORT": 0.3906, "train_start": "2000-11-16", "sampler": "SMOTETomek", "best_params": "{}", "features": ["XLY_Disc_vol_20d", "CAT_Caterpillar_ret_5d__macross__VRTX_VertexPharm_zscore_60d", "NFCI_ret_5d", "EWA_Australia_zscore_60d", "BAX_BankBoston_vol_20d__div__VRTX_VertexPharm_zscore_60d", "MSTR_Bitcoin3_ret_1d", "vix_vs_ma10__div__vix_zscore_10d", "AMT_AmericanTower_ret_1d", "Brent_Oil_FRED_ret_5d", "vix_zscore_10d__zrel__AMZN_zscore_60d", "vix_zscore_10d__div__spx_abs_ret_max_5d", "NFCI_ret_5d__div__CPB_CampbellSoup_vol_20d", "CPB_CampbellSoup_zscore_60d", "SLB_Schlumberger_ret_1d", "HD_ret_1d", "vix_vs_ma10__minus__CAT_Caterpillar_ret_5d", "vix_acceleration_1d"], "is_new": false}, {"model_id": "v1_h5_CALM_GradientBoosting_N6", "algo": "GradientBoosting", "regime": "CALM", "horizon": 5, "n_features": 6, "F1_dir": 0.5213, "F1_UP_FORT": 0.0976, "F1_DOWN_FORT": 0.3167, "train_start": "2000-11-16", "sampler": "SMOTETomek", "best_params": "{}", "features": ["XLY_Disc_vol_20d", "CAT_Caterpillar_ret_5d__macross__VRTX_VertexPharm_zscore_60d", "NFCI_ret_5d", "EWA_Australia_zscore_60d", "BAX_BankBoston_vol_20d__div__VRTX_VertexPharm_zscore_60d", "MSTR_Bitcoin3_ret_1d"], "is_new": false}, {"model_id": "v1_h5_CALM_GradientBoosting_N7", "algo": "GradientBoosting", "regime": "CALM", "horizon": 5, "n_features": 7, "F1_dir": 0.5641, "F1_UP_FORT": 0.1905, "F1_DOWN_FORT": 0.381, "train_start": "2000-11-16", "sampler": "SMOTETomek", "best_params": "{}", "features": ["XLY_Disc_vol_20d", "CAT_Caterpillar_ret_5d__macross__VRTX_VertexPharm_zscore_60d", "NFCI_ret_5d", "EWA_Australia_zscore_60d", "BAX_BankBoston_vol_20d__div__VRTX_VertexPharm_zscore_60d", "MSTR_Bitcoin3_ret_1d", "vix_vs_ma10__div__vix_zscore_10d"], "is_new": false}, {"model_id": "v1_h5_CALM_GradientBoosting_N8", "algo": "GradientBoosting", "regime": "CALM", "horizon": 5, "n_features": 8, "F1_dir": 0.5458, "F1_UP_FORT": 0.1839, "F1_DOWN_FORT": 0.3443, "train_start": "2000-11-16", "sampler": "SMOTETomek", "best_params": "{}", "features": ["XLY_Disc_vol_20d", "CAT_Caterpillar_ret_5d__macross__VRTX_VertexPharm_zscore_60d", "NFCI_ret_5d", "EWA_Australia_zscore_60d", "BAX_BankBoston_vol_20d__div__VRTX_VertexPharm_zscore_60d", "MSTR_Bitcoin3_ret_1d", "vix_vs_ma10__div__vix_zscore_10d", "AMT_AmericanTower_ret_1d"], "is_new": false}, {"model_id": "v1_h5_CALM_GradientBoosting_N9", "algo": "GradientBoosting", "regime": "CALM", "horizon": 5, "n_features": 9, "F1_dir": 0.5499, "F1_UP_FORT": 0.1839, "F1_DOWN_FORT": 0.3697, "train_start": "2000-11-16", "sampler": "SMOTETomek", "best_params": "{}", "features": ["XLY_Disc_vol_20d", "CAT_Caterpillar_ret_5d__macross__VRTX_VertexPharm_zscore_60d", "NFCI_ret_5d", "EWA_Australia_zscore_60d", "BAX_BankBoston_vol_20d__div__VRTX_VertexPharm_zscore_60d", "MSTR_Bitcoin3_ret_1d", "vix_vs_ma10__div__vix_zscore_10d", "AMT_AmericanTower_ret_1d", "Brent_Oil_FRED_ret_5d"], "is_new": false}, {"model_id": "v1_h5_CALM_GradientBoosting_N10", "algo": "GradientBoosting", "regime": "CALM", "horizon": 5, "n_features": 10, "F1_dir": 0.5877, "F1_UP_FORT": 0.2574, "F1_DOWN_FORT": 0.38, "train_start": "2000-11-16", "sampler": "SMOTETomek", "best_params": "{}", "features": ["XLY_Disc_vol_20d", "CAT_Caterpillar_ret_5d__macross__VRTX_VertexPharm_zscore_60d", "NFCI_ret_5d", "EWA_Australia_zscore_60d", "BAX_BankBoston_vol_20d__div__VRTX_VertexPharm_zscore_60d", "MSTR_Bitcoin3_ret_1d", "vix_vs_ma10__div__vix_zscore_10d", "AMT_AmericanTower_ret_1d", "Brent_Oil_FRED_ret_5d", "vix_zscore_10d__zrel__AMZN_zscore_60d"], "is_new": false}, {"model_id": "v1_h5_CALM_GradientBoosting_N11", "algo": "GradientBoosting", "regime": "CALM", "horizon": 5, "n_features": 11, "F1_dir": 0.5893, "F1_UP_FORT": 0.2174, "F1_DOWN_FORT": 0.3966, "train_start": "2000-11-16", "sampler": "SMOTETomek", "best_params": "{}", "features": ["XLY_Disc_vol_20d", "CAT_Caterpillar_ret_5d__macross__VRTX_VertexPharm_zscore_60d", "NFCI_ret_5d", "EWA_Australia_zscore_60d", "BAX_BankBoston_vol_20d__div__VRTX_VertexPharm_zscore_60d", "MSTR_Bitcoin3_ret_1d", "vix_vs_ma10__div__vix_zscore_10d", "AMT_AmericanTower_ret_1d", "Brent_Oil_FRED_ret_5d", "vix_zscore_10d__zrel__AMZN_zscore_60d", "vix_zscore_10d__div__spx_abs_ret_max_5d"], "is_new": false}, {"model_id": "v1_h5_CALM_GradientBoosting_N12", "algo": "GradientBoosting", "regime": "CALM", "horizon": 5, "n_features": 12, "F1_dir": 0.5609, "F1_UP_FORT": 0.1348, "F1_DOWN_FORT": 0.3902, "train_start": "2000-11-16", "sampler": "SMOTETomek", "best_params": "{}", "features": ["XLY_Disc_vol_20d", "CAT_Caterpillar_ret_5d__macross__VRTX_VertexPharm_zscore_60d", "NFCI_ret_5d", "EWA_Australia_zscore_60d", "BAX_BankBoston_vol_20d__div__VRTX_VertexPharm_zscore_60d", "MSTR_Bitcoin3_ret_1d", "vix_vs_ma10__div__vix_zscore_10d", "AMT_AmericanTower_ret_1d", "Brent_Oil_FRED_ret_5d", "vix_zscore_10d__zrel__AMZN_zscore_60d", "vix_zscore_10d__div__spx_abs_ret_max_5d", "NFCI_ret_5d__div__CPB_CampbellSoup_vol_20d"], "is_new": false}, {"model_id": "v1_h5_CALM_GradientBoosting_N13", "algo": "GradientBoosting", "regime": "CALM", "horizon": 5, "n_features": 13, "F1_dir": 0.5413, "F1_UP_FORT": 0.1609, "F1_DOWN_FORT": 0.3704, "train_start": "2000-11-16", "sampler": "SMOTETomek", "best_params": "{}", "features": ["XLY_Disc_vol_20d", "CAT_Caterpillar_ret_5d__macross__VRTX_VertexPharm_zscore_60d", "NFCI_ret_5d", "EWA_Australia_zscore_60d", "BAX_BankBoston_vol_20d__div__VRTX_VertexPharm_zscore_60d", "MSTR_Bitcoin3_ret_1d", "vix_vs_ma10__div__vix_zscore_10d", "AMT_AmericanTower_ret_1d", "Brent_Oil_FRED_ret_5d", "vix_zscore_10d__zrel__AMZN_zscore_60d", "vix_zscore_10d__div__spx_abs_ret_max_5d", "NFCI_ret_5d__div__CPB_CampbellSoup_vol_20d", "CPB_CampbellSoup_zscore_60d"], "is_new": false}, {"model_id": "v1_h5_CALM_GradientBoosting_N14", "algo": "GradientBoosting", "regime": "CALM", "horizon": 5, "n_features": 14, "F1_dir": 0.5934, "F1_UP_FORT": 0.1647, "F1_DOWN_FORT": 0.4225, "train_start": "2000-11-16", "sampler": "SMOTETomek", "best_params": "{}", "features": ["XLY_Disc_vol_20d", "CAT_Caterpillar_ret_5d__macross__VRTX_VertexPharm_zscore_60d", "NFCI_ret_5d", "EWA_Australia_zscore_60d", "BAX_BankBoston_vol_20d__div__VRTX_VertexPharm_zscore_60d", "MSTR_Bitcoin3_ret_1d", "vix_vs_ma10__div__vix_zscore_10d", "AMT_AmericanTower_ret_1d", "Brent_Oil_FRED_ret_5d", "vix_zscore_10d__zrel__AMZN_zscore_60d", "vix_zscore_10d__div__spx_abs_ret_max_5d", "NFCI_ret_5d__div__CPB_CampbellSoup_vol_20d", "CPB_CampbellSoup_zscore_60d", "SLB_Schlumberger_ret_1d"], "is_new": false}, {"model_id": "v1_h5_CALM_GradientBoosting_N15", "algo": "GradientBoosting", "regime": "CALM", "horizon": 5, "n_features": 15, "F1_dir": 0.5721, "F1_UP_FORT": 0.2174, "F1_DOWN_FORT": 0.3893, "train_start": "2000-11-16", "sampler": "SMOTETomek", "best_params": "{}", "features": ["XLY_Disc_vol_20d", "CAT_Caterpillar_ret_5d__macross__VRTX_VertexPharm_zscore_60d", "NFCI_ret_5d", "EWA_Australia_zscore_60d", "BAX_BankBoston_vol_20d__div__VRTX_VertexPharm_zscore_60d", "MSTR_Bitcoin3_ret_1d", "vix_vs_ma10__div__vix_zscore_10d", "AMT_AmericanTower_ret_1d", "Brent_Oil_FRED_ret_5d", "vix_zscore_10d__zrel__AMZN_zscore_60d", "vix_zscore_10d__div__spx_abs_ret_max_5d", "NFCI_ret_5d__div__CPB_CampbellSoup_vol_20d", "CPB_CampbellSoup_zscore_60d", "SLB_Schlumberger_ret_1d", "HD_ret_1d"], "is_new": false}, {"model_id": "v1_h5_CALM_GradientBoosting_N16", "algo": "GradientBoosting", "regime": "CALM", "horizon": 5, "n_features": 16, "F1_dir": 0.563, "F1_UP_FORT": 0.1758, "F1_DOWN_FORT": 0.3889, "train_start": "2000-11-16", "sampler": "SMOTETomek", "best_params": "{}", "features": ["XLY_Disc_vol_20d", "CAT_Caterpillar_ret_5d__macross__VRTX_VertexPharm_zscore_60d", "NFCI_ret_5d", "EWA_Australia_zscore_60d", "BAX_BankBoston_vol_20d__div__VRTX_VertexPharm_zscore_60d", "MSTR_Bitcoin3_ret_1d", "vix_vs_ma10__div__vix_zscore_10d", "AMT_AmericanTower_ret_1d", "Brent_Oil_FRED_ret_5d", "vix_zscore_10d__zrel__AMZN_zscore_60d", "vix_zscore_10d__div__spx_abs_ret_max_5d", "NFCI_ret_5d__div__CPB_CampbellSoup_vol_20d", "CPB_CampbellSoup_zscore_60d", "SLB_Schlumberger_ret_1d", "HD_ret_1d", "vix_vs_ma10__minus__CAT_Caterpillar_ret_5d"], "is_new": false}, {"model_id": "v1_h5_CALM_GradientBoosting_N17", "algo": "GradientBoosting", "regime": "CALM", "horizon": 5, "n_features": 17, "F1_dir": 0.608, "F1_UP_FORT": 0.1957, "F1_DOWN_FORT": 0.3906, "train_start": "2000-11-16", "sampler": "SMOTETomek", "best_params": "{}", "features": ["XLY_Disc_vol_20d", "CAT_Caterpillar_ret_5d__macross__VRTX_VertexPharm_zscore_60d", "NFCI_ret_5d", "EWA_Australia_zscore_60d", "BAX_BankBoston_vol_20d__div__VRTX_VertexPharm_zscore_60d", "MSTR_Bitcoin3_ret_1d", "vix_vs_ma10__div__vix_zscore_10d", "AMT_AmericanTower_ret_1d", "Brent_Oil_FRED_ret_5d", "vix_zscore_10d__zrel__AMZN_zscore_60d", "vix_zscore_10d__div__spx_abs_ret_max_5d", "NFCI_ret_5d__div__CPB_CampbellSoup_vol_20d", "CPB_CampbellSoup_zscore_60d", "SLB_Schlumberger_ret_1d", "HD_ret_1d", "vix_vs_ma10__minus__CAT_Caterpillar_ret_5d", "vix_acceleration_1d"], "is_new": false}, {"model_id": "v1_h5_CALM_RandomForest_N6", "algo": "RandomForest", "regime": "CALM", "horizon": 5, "n_features": 6, "F1_dir": 0.5513, "F1_UP_FORT": 0.1935, "F1_DOWN_FORT": 0.3902, "train_start": "2000-11-16", "sampler": "SMOTETomek", "best_params": "{}", "features": ["XLY_Disc_vol_20d", "CAT_Caterpillar_ret_5d__macross__VRTX_VertexPharm_zscore_60d", "NFCI_ret_5d", "EWA_Australia_zscore_60d", "BAX_BankBoston_vol_20d__div__VRTX_VertexPharm_zscore_60d", "MSTR_Bitcoin3_ret_1d"], "is_new": false}, {"model_id": "v1_h5_CALM_RandomForest_N7", "algo": "RandomForest", "regime": "CALM", "horizon": 5, "n_features": 7, "F1_dir": 0.5553, "F1_UP_FORT": 0.1818, "F1_DOWN_FORT": 0.4275, "train_start": "2000-11-16", "sampler": "SMOTETomek", "best_params": "{}", "features": ["XLY_Disc_vol_20d", "CAT_Caterpillar_ret_5d__macross__VRTX_VertexPharm_zscore_60d", "NFCI_ret_5d", "EWA_Australia_zscore_60d", "BAX_BankBoston_vol_20d__div__VRTX_VertexPharm_zscore_60d", "MSTR_Bitcoin3_ret_1d", "vix_vs_ma10__div__vix_zscore_10d"], "is_new": false}, {"model_id": "v1_h5_CALM_RandomForest_N8", "algo": "RandomForest", "regime": "CALM", "horizon": 5, "n_features": 8, "F1_dir": 0.5726, "F1_UP_FORT": 0.1935, "F1_DOWN_FORT": 0.3582, "train_start": "2000-11-16", "sampler": "SMOTETomek", "best_params": "{}", "features": ["XLY_Disc_vol_20d", "CAT_Caterpillar_ret_5d__macross__VRTX_VertexPharm_zscore_60d", "NFCI_ret_5d", "EWA_Australia_zscore_60d", "BAX_BankBoston_vol_20d__div__VRTX_VertexPharm_zscore_60d", "MSTR_Bitcoin3_ret_1d", "vix_vs_ma10__div__vix_zscore_10d", "AMT_AmericanTower_ret_1d"], "is_new": false}, {"model_id": "v1_h5_CALM_RandomForest_N9", "algo": "RandomForest", "regime": "CALM", "horizon": 5, "n_features": 9, "F1_dir": 0.5726, "F1_UP_FORT": 0.2174, "F1_DOWN_FORT": 0.4031, "train_start": "2000-11-16", "sampler": "SMOTETomek", "best_params": "{}", "features": ["XLY_Disc_vol_20d", "CAT_Caterpillar_ret_5d__macross__VRTX_VertexPharm_zscore_60d", "NFCI_ret_5d", "EWA_Australia_zscore_60d", "BAX_BankBoston_vol_20d__div__VRTX_VertexPharm_zscore_60d", "MSTR_Bitcoin3_ret_1d", "vix_vs_ma10__div__vix_zscore_10d", "AMT_AmericanTower_ret_1d", "Brent_Oil_FRED_ret_5d"], "is_new": false}, {"model_id": "v1_h5_CALM_RandomForest_N10", "algo": "RandomForest", "regime": "CALM", "horizon": 5, "n_features": 10, "F1_dir": 0.5598, "F1_UP_FORT": 0.2245, "F1_DOWN_FORT": 0.413, "train_start": "2000-11-16", "sampler": "SMOTETomek", "best_params": "{}", "features": ["XLY_Disc_vol_20d", "CAT_Caterpillar_ret_5d__macross__VRTX_VertexPharm_zscore_60d", "NFCI_ret_5d", "EWA_Australia_zscore_60d", "BAX_BankBoston_vol_20d__div__VRTX_VertexPharm_zscore_60d", "MSTR_Bitcoin3_ret_1d", "vix_vs_ma10__div__vix_zscore_10d", "AMT_AmericanTower_ret_1d", "Brent_Oil_FRED_ret_5d", "vix_zscore_10d__zrel__AMZN_zscore_60d"], "is_new": false}, {"model_id": "v1_h5_CALM_RandomForest_N11", "algo": "RandomForest", "regime": "CALM", "horizon": 5, "n_features": 11, "F1_dir": 0.5555, "F1_UP_FORT": 0.2391, "F1_DOWN_FORT": 0.4314, "train_start": "2000-11-16", "sampler": "SMOTETomek", "best_params": "{}", "features": ["XLY_Disc_vol_20d", "CAT_Caterpillar_ret_5d__macross__VRTX_VertexPharm_zscore_60d", "NFCI_ret_5d", "EWA_Australia_zscore_60d", "BAX_BankBoston_vol_20d__div__VRTX_VertexPharm_zscore_60d", "MSTR_Bitcoin3_ret_1d", "vix_vs_ma10__div__vix_zscore_10d", "AMT_AmericanTower_ret_1d", "Brent_Oil_FRED_ret_5d", "vix_zscore_10d__zrel__AMZN_zscore_60d", "vix_zscore_10d__div__spx_abs_ret_max_5d"], "is_new": false}, {"model_id": "v1_h5_CALM_RandomForest_N12", "algo": "RandomForest", "regime": "CALM", "horizon": 5, "n_features": 12, "F1_dir": 0.5443, "F1_UP_FORT": 0.1758, "F1_DOWN_FORT": 0.4174, "train_start": "2000-11-16", "sampler": "SMOTETomek", "best_params": "{}", "features": ["XLY_Disc_vol_20d", "CAT_Caterpillar_ret_5d__macross__VRTX_VertexPharm_zscore_60d", "NFCI_ret_5d", "EWA_Australia_zscore_60d", "BAX_BankBoston_vol_20d__div__VRTX_VertexPharm_zscore_60d", "MSTR_Bitcoin3_ret_1d", "vix_vs_ma10__div__vix_zscore_10d", "AMT_AmericanTower_ret_1d", "Brent_Oil_FRED_ret_5d", "vix_zscore_10d__zrel__AMZN_zscore_60d", "vix_zscore_10d__div__spx_abs_ret_max_5d", "NFCI_ret_5d__div__CPB_CampbellSoup_vol_20d"], "is_new": false}, {"model_id": "v1_h5_CALM_RandomForest_N13", "algo": "RandomForest", "regime": "CALM", "horizon": 5, "n_features": 13, "F1_dir": 0.5711, "F1_UP_FORT": 0.2553, "F1_DOWN_FORT": 0.4404, "train_start": "2000-11-16", "sampler": "SMOTETomek", "best_params": "{}", "features": ["XLY_Disc_vol_20d", "CAT_Caterpillar_ret_5d__macross__VRTX_VertexPharm_zscore_60d", "NFCI_ret_5d", "EWA_Australia_zscore_60d", "BAX_BankBoston_vol_20d__div__VRTX_VertexPharm_zscore_60d", "MSTR_Bitcoin3_ret_1d", "vix_vs_ma10__div__vix_zscore_10d", "AMT_AmericanTower_ret_1d", "Brent_Oil_FRED_ret_5d", "vix_zscore_10d__zrel__AMZN_zscore_60d", "vix_zscore_10d__div__spx_abs_ret_max_5d", "NFCI_ret_5d__div__CPB_CampbellSoup_vol_20d", "CPB_CampbellSoup_zscore_60d"], "is_new": false}, {"model_id": "v1_h5_CALM_RandomForest_N14", "algo": "RandomForest", "regime": "CALM", "horizon": 5, "n_features": 14, "F1_dir": 0.5499, "F1_UP_FORT": 0.2316, "F1_DOWN_FORT": 0.4522, "train_start": "2000-11-16", "sampler": "SMOTETomek", "best_params": "{}", "features": ["XLY_Disc_vol_20d", "CAT_Caterpillar_ret_5d__macross__VRTX_VertexPharm_zscore_60d", "NFCI_ret_5d", "EWA_Australia_zscore_60d", "BAX_BankBoston_vol_20d__div__VRTX_VertexPharm_zscore_60d", "MSTR_Bitcoin3_ret_1d", "vix_vs_ma10__div__vix_zscore_10d", "AMT_AmericanTower_ret_1d", "Brent_Oil_FRED_ret_5d", "vix_zscore_10d__zrel__AMZN_zscore_60d", "vix_zscore_10d__div__spx_abs_ret_max_5d", "NFCI_ret_5d__div__CPB_CampbellSoup_vol_20d", "CPB_CampbellSoup_zscore_60d", "SLB_Schlumberger_ret_1d"], "is_new": false}, {"model_id": "v1_h5_CALM_RandomForest_N15", "algo": "RandomForest", "regime": "CALM", "horizon": 5, "n_features": 15, "F1_dir": 0.5763, "F1_UP_FORT": 0.2553, "F1_DOWN_FORT": 0.4425, "train_start": "2000-11-16", "sampler": "SMOTETomek", "best_params": "{}", "features": ["XLY_Disc_vol_20d", "CAT_Caterpillar_ret_5d__macross__VRTX_VertexPharm_zscore_60d", "NFCI_ret_5d", "EWA_Australia_zscore_60d", "BAX_BankBoston_vol_20d__div__VRTX_VertexPharm_zscore_60d", "MSTR_Bitcoin3_ret_1d", "vix_vs_ma10__div__vix_zscore_10d", "AMT_AmericanTower_ret_1d", "Brent_Oil_FRED_ret_5d", "vix_zscore_10d__zrel__AMZN_zscore_60d", "vix_zscore_10d__div__spx_abs_ret_max_5d", "NFCI_ret_5d__div__CPB_CampbellSoup_vol_20d", "CPB_CampbellSoup_zscore_60d", "SLB_Schlumberger_ret_1d", "HD_ret_1d"], "is_new": false}, {"model_id": "v1_h5_CALM_RandomForest_N16", "algo": "RandomForest", "regime": "CALM", "horizon": 5, "n_features": 16, "F1_dir": 0.5376, "F1_UP_FORT": 0.1591, "F1_DOWN_FORT": 0.4074, "train_start": "2000-11-16", "sampler": "SMOTETomek", "best_params": "{}", "features": ["XLY_Disc_vol_20d", "CAT_Caterpillar_ret_5d__macross__VRTX_VertexPharm_zscore_60d", "NFCI_ret_5d", "EWA_Australia_zscore_60d", "BAX_BankBoston_vol_20d__div__VRTX_VertexPharm_zscore_60d", "MSTR_Bitcoin3_ret_1d", "vix_vs_ma10__div__vix_zscore_10d", "AMT_AmericanTower_ret_1d", "Brent_Oil_FRED_ret_5d", "vix_zscore_10d__zrel__AMZN_zscore_60d", "vix_zscore_10d__div__spx_abs_ret_max_5d", "NFCI_ret_5d__div__CPB_CampbellSoup_vol_20d", "CPB_CampbellSoup_zscore_60d", "SLB_Schlumberger_ret_1d", "HD_ret_1d", "vix_vs_ma10__minus__CAT_Caterpillar_ret_5d"], "is_new": false}, {"model_id": "v1_h5_CALM_RandomForest_N17", "algo": "RandomForest", "regime": "CALM", "horizon": 5, "n_features": 17, "F1_dir": 0.5625, "F1_UP_FORT": 0.1978, "F1_DOWN_FORT": 0.4144, "train_start": "2000-11-16", "sampler": "SMOTETomek", "best_params": "{}", "features": ["XLY_Disc_vol_20d", "CAT_Caterpillar_ret_5d__macross__VRTX_VertexPharm_zscore_60d", "NFCI_ret_5d", "EWA_Australia_zscore_60d", "BAX_BankBoston_vol_20d__div__VRTX_VertexPharm_zscore_60d", "MSTR_Bitcoin3_ret_1d", "vix_vs_ma10__div__vix_zscore_10d", "AMT_AmericanTower_ret_1d", "Brent_Oil_FRED_ret_5d", "vix_zscore_10d__zrel__AMZN_zscore_60d", "vix_zscore_10d__div__spx_abs_ret_max_5d", "NFCI_ret_5d__div__CPB_CampbellSoup_vol_20d", "CPB_CampbellSoup_zscore_60d", "SLB_Schlumberger_ret_1d", "HD_ret_1d", "vix_vs_ma10__minus__CAT_Caterpillar_ret_5d", "vix_acceleration_1d"], "is_new": false}, {"model_id": "v1_h5_CALM_LogisticRegression_N6", "algo": "LogisticRegression", "regime": "CALM", "horizon": 5, "n_features": 6, "F1_dir": 0.5081, "F1_UP_FORT": 0.1364, "F1_DOWN_FORT": 0.315, "train_start": "2000-11-16", "sampler": "SMOTETomek", "best_params": "{}", "features": ["XLY_Disc_vol_20d", "CAT_Caterpillar_ret_5d__macross__VRTX_VertexPharm_zscore_60d", "NFCI_ret_5d", "EWA_Australia_zscore_60d", "BAX_BankBoston_vol_20d__div__VRTX_VertexPharm_zscore_60d", "MSTR_Bitcoin3_ret_1d"], "is_new": false}, {"model_id": "v1_h5_CALM_LogisticRegression_N7", "algo": "LogisticRegression", "regime": "CALM", "horizon": 5, "n_features": 7, "F1_dir": 0.5128, "F1_UP_FORT": 0.1739, "F1_DOWN_FORT": 0.3193, "train_start": "2000-11-16", "sampler": "SMOTETomek", "best_params": "{}", "features": ["XLY_Disc_vol_20d", "CAT_Caterpillar_ret_5d__macross__VRTX_VertexPharm_zscore_60d", "NFCI_ret_5d", "EWA_Australia_zscore_60d", "BAX_BankBoston_vol_20d__div__VRTX_VertexPharm_zscore_60d", "MSTR_Bitcoin3_ret_1d", "vix_vs_ma10__div__vix_zscore_10d"], "is_new": false}, {"model_id": "v1_h5_CALM_LogisticRegression_N8", "algo": "LogisticRegression", "regime": "CALM", "horizon": 5, "n_features": 8, "F1_dir": 0.5, "F1_UP_FORT": 0.1915, "F1_DOWN_FORT": 0.2857, "train_start": "2000-11-16", "sampler": "SMOTETomek", "best_params": "{}", "features": ["XLY_Disc_vol_20d", "CAT_Caterpillar_ret_5d__macross__VRTX_VertexPharm_zscore_60d", "NFCI_ret_5d", "EWA_Australia_zscore_60d", "BAX_BankBoston_vol_20d__div__VRTX_VertexPharm_zscore_60d", "MSTR_Bitcoin3_ret_1d", "vix_vs_ma10__div__vix_zscore_10d", "AMT_AmericanTower_ret_1d"], "is_new": false}, {"model_id": "v1_h5_CALM_LogisticRegression_N9", "algo": "LogisticRegression", "regime": "CALM", "horizon": 5, "n_features": 9, "F1_dir": 0.5085, "F1_UP_FORT": 0.1957, "F1_DOWN_FORT": 0.281, "train_start": "2000-11-16", "sampler": "SMOTETomek", "best_params": "{}", "features": ["XLY_Disc_vol_20d", "CAT_Caterpillar_ret_5d__macross__VRTX_VertexPharm_zscore_60d", "NFCI_ret_5d", "EWA_Australia_zscore_60d", "BAX_BankBoston_vol_20d__div__VRTX_VertexPharm_zscore_60d", "MSTR_Bitcoin3_ret_1d", "vix_vs_ma10__div__vix_zscore_10d", "AMT_AmericanTower_ret_1d", "Brent_Oil_FRED_ret_5d"], "is_new": false}, {"model_id": "v1_h5_CALM_LogisticRegression_N10", "algo": "LogisticRegression", "regime": "CALM", "horizon": 5, "n_features": 10, "F1_dir": 0.5462, "F1_UP_FORT": 0.2115, "F1_DOWN_FORT": 0.35, "train_start": "2000-11-16", "sampler": "SMOTETomek", "best_params": "{}", "features": ["XLY_Disc_vol_20d", "CAT_Caterpillar_ret_5d__macross__VRTX_VertexPharm_zscore_60d", "NFCI_ret_5d", "EWA_Australia_zscore_60d", "BAX_BankBoston_vol_20d__div__VRTX_VertexPharm_zscore_60d", "MSTR_Bitcoin3_ret_1d", "vix_vs_ma10__div__vix_zscore_10d", "AMT_AmericanTower_ret_1d", "Brent_Oil_FRED_ret_5d", "vix_zscore_10d__zrel__AMZN_zscore_60d"], "is_new": false}, {"model_id": "v1_h5_CALM_LogisticRegression_N11", "algo": "LogisticRegression", "regime": "CALM", "horizon": 5, "n_features": 11, "F1_dir": 0.5462, "F1_UP_FORT": 0.1837, "F1_DOWN_FORT": 0.4628, "train_start": "2000-11-16", "sampler": "SMOTETomek", "best_params": "{}", "features": ["XLY_Disc_vol_20d", "CAT_Caterpillar_ret_5d__macross__VRTX_VertexPharm_zscore_60d", "NFCI_ret_5d", "EWA_Australia_zscore_60d", "BAX_BankBoston_vol_20d__div__VRTX_VertexPharm_zscore_60d", "MSTR_Bitcoin3_ret_1d", "vix_vs_ma10__div__vix_zscore_10d", "AMT_AmericanTower_ret_1d", "Brent_Oil_FRED_ret_5d", "vix_zscore_10d__zrel__AMZN_zscore_60d", "vix_zscore_10d__div__spx_abs_ret_max_5d"], "is_new": false}, {"model_id": "v1_h5_CALM_LogisticRegression_N12", "algo": "LogisticRegression", "regime": "CALM", "horizon": 5, "n_features": 12, "F1_dir": 0.5506, "F1_UP_FORT": 0.1837, "F1_DOWN_FORT": 0.4444, "train_start": "2000-11-16", "sampler": "SMOTETomek", "best_params": "{}", "features": ["XLY_Disc_vol_20d", "CAT_Caterpillar_ret_5d__macross__VRTX_VertexPharm_zscore_60d", "NFCI_ret_5d", "EWA_Australia_zscore_60d", "BAX_BankBoston_vol_20d__div__VRTX_VertexPharm_zscore_60d", "MSTR_Bitcoin3_ret_1d", "vix_vs_ma10__div__vix_zscore_10d", "AMT_AmericanTower_ret_1d", "Brent_Oil_FRED_ret_5d", "vix_zscore_10d__zrel__AMZN_zscore_60d", "vix_zscore_10d__div__spx_abs_ret_max_5d", "NFCI_ret_5d__div__CPB_CampbellSoup_vol_20d"], "is_new": false}, {"model_id": "v1_h5_CALM_LogisticRegression_N13", "algo": "LogisticRegression", "regime": "CALM", "horizon": 5, "n_features": 13, "F1_dir": 0.5291, "F1_UP_FORT": 0.2041, "F1_DOWN_FORT": 0.4486, "train_start": "2000-11-16", "sampler": "SMOTETomek", "best_params": "{}", "features": ["XLY_Disc_vol_20d", "CAT_Caterpillar_ret_5d__macross__VRTX_VertexPharm_zscore_60d", "NFCI_ret_5d", "EWA_Australia_zscore_60d", "BAX_BankBoston_vol_20d__div__VRTX_VertexPharm_zscore_60d", "MSTR_Bitcoin3_ret_1d", "vix_vs_ma10__div__vix_zscore_10d", "AMT_AmericanTower_ret_1d", "Brent_Oil_FRED_ret_5d", "vix_zscore_10d__zrel__AMZN_zscore_60d", "vix_zscore_10d__div__spx_abs_ret_max_5d", "NFCI_ret_5d__div__CPB_CampbellSoup_vol_20d", "CPB_CampbellSoup_zscore_60d"], "is_new": false}, {"model_id": "v1_h5_CALM_LogisticRegression_N14", "algo": "LogisticRegression", "regime": "CALM", "horizon": 5, "n_features": 14, "F1_dir": 0.5335, "F1_UP_FORT": 0.2, "F1_DOWN_FORT": 0.4259, "train_start": "2000-11-16", "sampler": "SMOTETomek", "best_params": "{}", "features": ["XLY_Disc_vol_20d", "CAT_Caterpillar_ret_5d__macross__VRTX_VertexPharm_zscore_60d", "NFCI_ret_5d", "EWA_Australia_zscore_60d", "BAX_BankBoston_vol_20d__div__VRTX_VertexPharm_zscore_60d", "MSTR_Bitcoin3_ret_1d", "vix_vs_ma10__div__vix_zscore_10d", "AMT_AmericanTower_ret_1d", "Brent_Oil_FRED_ret_5d", "vix_zscore_10d__zrel__AMZN_zscore_60d", "vix_zscore_10d__div__spx_abs_ret_max_5d", "NFCI_ret_5d__div__CPB_CampbellSoup_vol_20d", "CPB_CampbellSoup_zscore_60d", "SLB_Schlumberger_ret_1d"], "is_new": false}, {"model_id": "v1_h5_CALM_LogisticRegression_N15", "algo": "LogisticRegression", "regime": "CALM", "horizon": 5, "n_features": 15, "F1_dir": 0.5383, "F1_UP_FORT": 0.1818, "F1_DOWN_FORT": 0.434, "train_start": "2000-11-16", "sampler": "SMOTETomek", "best_params": "{}", "features": ["XLY_Disc_vol_20d", "CAT_Caterpillar_ret_5d__macross__VRTX_VertexPharm_zscore_60d", "NFCI_ret_5d", "EWA_Australia_zscore_60d", "BAX_BankBoston_vol_20d__div__VRTX_VertexPharm_zscore_60d", "MSTR_Bitcoin3_ret_1d", "vix_vs_ma10__div__vix_zscore_10d", "AMT_AmericanTower_ret_1d", "Brent_Oil_FRED_ret_5d", "vix_zscore_10d__zrel__AMZN_zscore_60d", "vix_zscore_10d__div__spx_abs_ret_max_5d", "NFCI_ret_5d__div__CPB_CampbellSoup_vol_20d", "CPB_CampbellSoup_zscore_60d", "SLB_Schlumberger_ret_1d", "HD_ret_1d"], "is_new": false}, {"model_id": "v1_h5_CALM_LogisticRegression_N16", "algo": "LogisticRegression", "regime": "CALM", "horizon": 5, "n_features": 16, "F1_dir": 0.5372, "F1_UP_FORT": 0.1633, "F1_DOWN_FORT": 0.4333, "train_start": "2000-11-16", "sampler": "SMOTETomek", "best_params": "{}", "features": ["XLY_Disc_vol_20d", "CAT_Caterpillar_ret_5d__macross__VRTX_VertexPharm_zscore_60d", "NFCI_ret_5d", "EWA_Australia_zscore_60d", "BAX_BankBoston_vol_20d__div__VRTX_VertexPharm_zscore_60d", "MSTR_Bitcoin3_ret_1d", "vix_vs_ma10__div__vix_zscore_10d", "AMT_AmericanTower_ret_1d", "Brent_Oil_FRED_ret_5d", "vix_zscore_10d__zrel__AMZN_zscore_60d", "vix_zscore_10d__div__spx_abs_ret_max_5d", "NFCI_ret_5d__div__CPB_CampbellSoup_vol_20d", "CPB_CampbellSoup_zscore_60d", "SLB_Schlumberger_ret_1d", "HD_ret_1d", "vix_vs_ma10__minus__CAT_Caterpillar_ret_5d"], "is_new": false}, {"model_id": "v1_h5_CALM_LogisticRegression_N17", "algo": "LogisticRegression", "regime": "CALM", "horizon": 5, "n_features": 17, "F1_dir": 0.5553, "F1_UP_FORT": 0.1633, "F1_DOWN_FORT": 0.4202, "train_start": "2000-11-16", "sampler": "SMOTETomek", "best_params": "{}", "features": ["XLY_Disc_vol_20d", "CAT_Caterpillar_ret_5d__macross__VRTX_VertexPharm_zscore_60d", "NFCI_ret_5d", "EWA_Australia_zscore_60d", "BAX_BankBoston_vol_20d__div__VRTX_VertexPharm_zscore_60d", "MSTR_Bitcoin3_ret_1d", "vix_vs_ma10__div__vix_zscore_10d", "AMT_AmericanTower_ret_1d", "Brent_Oil_FRED_ret_5d", "vix_zscore_10d__zrel__AMZN_zscore_60d", "vix_zscore_10d__div__spx_abs_ret_max_5d", "NFCI_ret_5d__div__CPB_CampbellSoup_vol_20d", "CPB_CampbellSoup_zscore_60d", "SLB_Schlumberger_ret_1d", "HD_ret_1d", "vix_vs_ma10__minus__CAT_Caterpillar_ret_5d", "vix_acceleration_1d"], "is_new": false}, {"model_id": "v1_h5_CALM_GradientBoosting_Optuna_N10", "algo": "GradientBoosting", "regime": "CALM", "horizon": 5, "n_features": 10, "F1_dir": 0.6164, "F1_UP_FORT": 0.3265, "F1_DOWN_FORT": 0.3579, "train_start": "2000-11-16", "sampler": "SMOTETomek", "best_params": "{}", "features": ["XLY_Disc_vol_20d", "CAT_Caterpillar_ret_5d__macross__VRTX_VertexPharm_zscore_60d", "NFCI_ret_5d", "EWA_Australia_zscore_60d", "BAX_BankBoston_vol_20d__div__VRTX_VertexPharm_zscore_60d", "MSTR_Bitcoin3_ret_1d", "vix_vs_ma10__div__vix_zscore_10d", "AMT_AmericanTower_ret_1d", "Brent_Oil_FRED_ret_5d", "vix_zscore_10d__zrel__AMZN_zscore_60d"], "is_new": false}, {"model_id": "v1_h5_CALM_GradientBoosting_OptunaCal_N10", "algo": "GradientBoostingCal", "regime": "CALM", "horizon": 5, "n_features": 10, "F1_dir": 0.6047, "F1_UP_FORT": 0.1333, "F1_DOWN_FORT": 0.3802, "train_start": "2000-11-16", "sampler": "SMOTETomek", "best_params": "{}", "features": ["XLY_Disc_vol_20d", "CAT_Caterpillar_ret_5d__macross__VRTX_VertexPharm_zscore_60d", "NFCI_ret_5d", "EWA_Australia_zscore_60d", "BAX_BankBoston_vol_20d__div__VRTX_VertexPharm_zscore_60d", "MSTR_Bitcoin3_ret_1d", "vix_vs_ma10__div__vix_zscore_10d", "AMT_AmericanTower_ret_1d", "Brent_Oil_FRED_ret_5d", "vix_zscore_10d__zrel__AMZN_zscore_60d"], "is_new": false}, {"model_id": "v1_h5_NORMAL_XGBoost_N5", "algo": "XGBoost", "regime": "NORMAL", "horizon": 5, "n_features": 5, "F1_dir": 0.5147, "F1_UP_FORT": 0.2685, "F1_DOWN_FORT": 0.3022, "train_start": "2001-02-07", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["NFCI_ret_5d__minus__NFCI_vol_20d", "heston_var_ev_h1__prod__vix_max_abs_ret_5d", "BTI_BritishAmerican_ret_20d", "heston_var_ev_h1__minus__M_Macys_vol_20d", "Core_CPI_zscore_60d__div__CTAS_Cintas_vol_20d"], "is_new": false}, {"model_id": "v1_h5_NORMAL_XGBoost_N6", "algo": "XGBoost", "regime": "NORMAL", "horizon": 5, "n_features": 6, "F1_dir": 0.5243, "F1_UP_FORT": 0.3057, "F1_DOWN_FORT": 0.3135, "train_start": "2001-02-07", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["NFCI_ret_5d__minus__NFCI_vol_20d", "heston_var_ev_h1__prod__vix_max_abs_ret_5d", "BTI_BritishAmerican_ret_20d", "heston_var_ev_h1__minus__M_Macys_vol_20d", "Core_CPI_zscore_60d__div__CTAS_Cintas_vol_20d", "EWA_Australia_ret_1d"], "is_new": false}, {"model_id": "v1_h5_NORMAL_XGBoost_N7", "algo": "XGBoost", "regime": "NORMAL", "horizon": 5, "n_features": 7, "F1_dir": 0.5429, "F1_UP_FORT": 0.3093, "F1_DOWN_FORT": 0.3125, "train_start": "2001-02-07", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["NFCI_ret_5d__minus__NFCI_vol_20d", "heston_var_ev_h1__prod__vix_max_abs_ret_5d", "BTI_BritishAmerican_ret_20d", "heston_var_ev_h1__minus__M_Macys_vol_20d", "Core_CPI_zscore_60d__div__CTAS_Cintas_vol_20d", "EWA_Australia_ret_1d", "AVB_AvalonBay_zscore_60d"], "is_new": false}, {"model_id": "v1_h5_NORMAL_XGBoost_N8", "algo": "XGBoost", "regime": "NORMAL", "horizon": 5, "n_features": 8, "F1_dir": 0.5167, "F1_UP_FORT": 0.3158, "F1_DOWN_FORT": 0.3759, "train_start": "2001-02-07", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["NFCI_ret_5d__minus__NFCI_vol_20d", "heston_var_ev_h1__prod__vix_max_abs_ret_5d", "BTI_BritishAmerican_ret_20d", "heston_var_ev_h1__minus__M_Macys_vol_20d", "Core_CPI_zscore_60d__div__CTAS_Cintas_vol_20d", "EWA_Australia_ret_1d", "AVB_AvalonBay_zscore_60d", "heston_var_ev_h1__div__vix_zscore_10d"], "is_new": false}, {"model_id": "v1_h5_NORMAL_XGBoost_N9", "algo": "XGBoost", "regime": "NORMAL", "horizon": 5, "n_features": 9, "F1_dir": 0.5221, "F1_UP_FORT": 0.2967, "F1_DOWN_FORT": 0.3805, "train_start": "2001-02-07", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["NFCI_ret_5d__minus__NFCI_vol_20d", "heston_var_ev_h1__prod__vix_max_abs_ret_5d", "BTI_BritishAmerican_ret_20d", "heston_var_ev_h1__minus__M_Macys_vol_20d", "Core_CPI_zscore_60d__div__CTAS_Cintas_vol_20d", "EWA_Australia_ret_1d", "AVB_AvalonBay_zscore_60d", "heston_var_ev_h1__div__vix_zscore_10d", "HD_zscore_60d"], "is_new": false}, {"model_id": "v1_h5_NORMAL_XGBoost_N11", "algo": "XGBoost", "regime": "NORMAL", "horizon": 5, "n_features": 11, "F1_dir": 0.5035, "F1_UP_FORT": 0.2864, "F1_DOWN_FORT": 0.3325, "train_start": "2001-02-07", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["NFCI_ret_5d__minus__NFCI_vol_20d", "heston_var_ev_h1__prod__vix_max_abs_ret_5d", "BTI_BritishAmerican_ret_20d", "heston_var_ev_h1__minus__M_Macys_vol_20d", "Core_CPI_zscore_60d__div__CTAS_Cintas_vol_20d", "EWA_Australia_ret_1d", "AVB_AvalonBay_zscore_60d", "heston_var_ev_h1__div__vix_zscore_10d", "HD_zscore_60d", "heston_var_ev_h1__div__NFCI_vol_20d", "HUM_Humana_ret_5d"], "is_new": false}, {"model_id": "v1_h5_NORMAL_XGBoost_N12", "algo": "XGBoost", "regime": "NORMAL", "horizon": 5, "n_features": 12, "F1_dir": 0.5252, "F1_UP_FORT": 0.3163, "F1_DOWN_FORT": 0.3756, "train_start": "2001-02-07", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["NFCI_ret_5d__minus__NFCI_vol_20d", "heston_var_ev_h1__prod__vix_max_abs_ret_5d", "BTI_BritishAmerican_ret_20d", "heston_var_ev_h1__minus__M_Macys_vol_20d", "Core_CPI_zscore_60d__div__CTAS_Cintas_vol_20d", "EWA_Australia_ret_1d", "AVB_AvalonBay_zscore_60d", "heston_var_ev_h1__div__vix_zscore_10d", "HD_zscore_60d", "heston_var_ev_h1__div__NFCI_vol_20d", "HUM_Humana_ret_5d", "Core_CPI_zscore_60d__macross__CTAS_Cintas_vol_20d"], "is_new": false}, {"model_id": "v1_h5_NORMAL_XGBoost_N13", "algo": "XGBoost", "regime": "NORMAL", "horizon": 5, "n_features": 13, "F1_dir": 0.5089, "F1_UP_FORT": 0.3141, "F1_DOWN_FORT": 0.3521, "train_start": "2001-02-07", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["NFCI_ret_5d__minus__NFCI_vol_20d", "heston_var_ev_h1__prod__vix_max_abs_ret_5d", "BTI_BritishAmerican_ret_20d", "heston_var_ev_h1__minus__M_Macys_vol_20d", "Core_CPI_zscore_60d__div__CTAS_Cintas_vol_20d", "EWA_Australia_ret_1d", "AVB_AvalonBay_zscore_60d", "heston_var_ev_h1__div__vix_zscore_10d", "HD_zscore_60d", "heston_var_ev_h1__div__NFCI_vol_20d", "HUM_Humana_ret_5d", "Core_CPI_zscore_60d__macross__CTAS_Cintas_vol_20d", "NFCI_ret_5d__ret5x__CTAS_Cintas_vol_20d"], "is_new": false}, {"model_id": "v1_h5_NORMAL_XGBoost_N14", "algo": "XGBoost", "regime": "NORMAL", "horizon": 5, "n_features": 14, "F1_dir": 0.5078, "F1_UP_FORT": 0.2968, "F1_DOWN_FORT": 0.3507, "train_start": "2001-02-07", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["NFCI_ret_5d__minus__NFCI_vol_20d", "heston_var_ev_h1__prod__vix_max_abs_ret_5d", "BTI_BritishAmerican_ret_20d", "heston_var_ev_h1__minus__M_Macys_vol_20d", "Core_CPI_zscore_60d__div__CTAS_Cintas_vol_20d", "EWA_Australia_ret_1d", "AVB_AvalonBay_zscore_60d", "heston_var_ev_h1__div__vix_zscore_10d", "HD_zscore_60d", "heston_var_ev_h1__div__NFCI_vol_20d", "HUM_Humana_ret_5d", "Core_CPI_zscore_60d__macross__CTAS_Cintas_vol_20d", "NFCI_ret_5d__ret5x__CTAS_Cintas_vol_20d", "XLY_Disc_ret_5d__ret5x__CLX_Clorox_ret_1d"], "is_new": false}, {"model_id": "v1_h5_NORMAL_XGBoost_N15", "algo": "XGBoost", "regime": "NORMAL", "horizon": 5, "n_features": 15, "F1_dir": 0.5212, "F1_UP_FORT": 0.3061, "F1_DOWN_FORT": 0.3476, "train_start": "2001-02-07", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["NFCI_ret_5d__minus__NFCI_vol_20d", "heston_var_ev_h1__prod__vix_max_abs_ret_5d", "BTI_BritishAmerican_ret_20d", "heston_var_ev_h1__minus__M_Macys_vol_20d", "Core_CPI_zscore_60d__div__CTAS_Cintas_vol_20d", "EWA_Australia_ret_1d", "AVB_AvalonBay_zscore_60d", "heston_var_ev_h1__div__vix_zscore_10d", "HD_zscore_60d", "heston_var_ev_h1__div__NFCI_vol_20d", "HUM_Humana_ret_5d", "Core_CPI_zscore_60d__macross__CTAS_Cintas_vol_20d", "NFCI_ret_5d__ret5x__CTAS_Cintas_vol_20d", "XLY_Disc_ret_5d__ret5x__CLX_Clorox_ret_1d", "PG_ret_20d"], "is_new": false}, {"model_id": "v1_h5_NORMAL_XGBoost_N16", "algo": "XGBoost", "regime": "NORMAL", "horizon": 5, "n_features": 16, "F1_dir": 0.5166, "F1_UP_FORT": 0.3333, "F1_DOWN_FORT": 0.338, "train_start": "2001-02-07", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["NFCI_ret_5d__minus__NFCI_vol_20d", "heston_var_ev_h1__prod__vix_max_abs_ret_5d", "BTI_BritishAmerican_ret_20d", "heston_var_ev_h1__minus__M_Macys_vol_20d", "Core_CPI_zscore_60d__div__CTAS_Cintas_vol_20d", "EWA_Australia_ret_1d", "AVB_AvalonBay_zscore_60d", "heston_var_ev_h1__div__vix_zscore_10d", "HD_zscore_60d", "heston_var_ev_h1__div__NFCI_vol_20d", "HUM_Humana_ret_5d", "Core_CPI_zscore_60d__macross__CTAS_Cintas_vol_20d", "NFCI_ret_5d__ret5x__CTAS_Cintas_vol_20d", "XLY_Disc_ret_5d__ret5x__CLX_Clorox_ret_1d", "PG_ret_20d", "NFCI_ret_5d__prod__vix_zscore_10d"], "is_new": false}, {"model_id": "v1_h5_NORMAL_XGBoost_N17", "algo": "XGBoost", "regime": "NORMAL", "horizon": 5, "n_features": 17, "F1_dir": 0.5049, "F1_UP_FORT": 0.294, "F1_DOWN_FORT": 0.3801, "train_start": "2001-02-07", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["NFCI_ret_5d__minus__NFCI_vol_20d", "heston_var_ev_h1__prod__vix_max_abs_ret_5d", "BTI_BritishAmerican_ret_20d", "heston_var_ev_h1__minus__M_Macys_vol_20d", "Core_CPI_zscore_60d__div__CTAS_Cintas_vol_20d", "EWA_Australia_ret_1d", "AVB_AvalonBay_zscore_60d", "heston_var_ev_h1__div__vix_zscore_10d", "HD_zscore_60d", "heston_var_ev_h1__div__NFCI_vol_20d", "HUM_Humana_ret_5d", "Core_CPI_zscore_60d__macross__CTAS_Cintas_vol_20d", "NFCI_ret_5d__ret5x__CTAS_Cintas_vol_20d", "XLY_Disc_ret_5d__ret5x__CLX_Clorox_ret_1d", "PG_ret_20d", "NFCI_ret_5d__prod__vix_zscore_10d", "Nikkei_Japan_vol_20d"], "is_new": false}, {"model_id": "v1_h5_NORMAL_XGBoost_N18", "algo": "XGBoost", "regime": "NORMAL", "horizon": 5, "n_features": 18, "F1_dir": 0.506, "F1_UP_FORT": 0.288, "F1_DOWN_FORT": 0.381, "train_start": "2001-02-07", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["NFCI_ret_5d__minus__NFCI_vol_20d", "heston_var_ev_h1__prod__vix_max_abs_ret_5d", "BTI_BritishAmerican_ret_20d", "heston_var_ev_h1__minus__M_Macys_vol_20d", "Core_CPI_zscore_60d__div__CTAS_Cintas_vol_20d", "EWA_Australia_ret_1d", "AVB_AvalonBay_zscore_60d", "heston_var_ev_h1__div__vix_zscore_10d", "HD_zscore_60d", "heston_var_ev_h1__div__NFCI_vol_20d", "HUM_Humana_ret_5d", "Core_CPI_zscore_60d__macross__CTAS_Cintas_vol_20d", "NFCI_ret_5d__ret5x__CTAS_Cintas_vol_20d", "XLY_Disc_ret_5d__ret5x__CLX_Clorox_ret_1d", "PG_ret_20d", "NFCI_ret_5d__prod__vix_zscore_10d", "Nikkei_Japan_vol_20d", "CLX_Clorox_ret_1d__macross__EWH_HongKong_ret_5d"], "is_new": false}, {"model_id": "v1_h5_NORMAL_LightGBM_N5", "algo": "LightGBM", "regime": "NORMAL", "horizon": 5, "n_features": 5, "F1_dir": 0.5038, "F1_UP_FORT": 0.2182, "F1_DOWN_FORT": 0.254, "train_start": "2001-02-07", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["NFCI_ret_5d__minus__NFCI_vol_20d", "heston_var_ev_h1__prod__vix_max_abs_ret_5d", "BTI_BritishAmerican_ret_20d", "heston_var_ev_h1__minus__M_Macys_vol_20d", "Core_CPI_zscore_60d__div__CTAS_Cintas_vol_20d"], "is_new": false}, {"model_id": "v1_h5_NORMAL_LightGBM_N6", "algo": "LightGBM", "regime": "NORMAL", "horizon": 5, "n_features": 6, "F1_dir": 0.5346, "F1_UP_FORT": 0.2857, "F1_DOWN_FORT": 0.2872, "train_start": "2001-02-07", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["NFCI_ret_5d__minus__NFCI_vol_20d", "heston_var_ev_h1__prod__vix_max_abs_ret_5d", "BTI_BritishAmerican_ret_20d", "heston_var_ev_h1__minus__M_Macys_vol_20d", "Core_CPI_zscore_60d__div__CTAS_Cintas_vol_20d", "EWA_Australia_ret_1d"], "is_new": false}, {"model_id": "v1_h5_NORMAL_LightGBM_N7", "algo": "LightGBM", "regime": "NORMAL", "horizon": 5, "n_features": 7, "F1_dir": 0.5177, "F1_UP_FORT": 0.2782, "F1_DOWN_FORT": 0.291, "train_start": "2001-02-07", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["NFCI_ret_5d__minus__NFCI_vol_20d", "heston_var_ev_h1__prod__vix_max_abs_ret_5d", "BTI_BritishAmerican_ret_20d", "heston_var_ev_h1__minus__M_Macys_vol_20d", "Core_CPI_zscore_60d__div__CTAS_Cintas_vol_20d", "EWA_Australia_ret_1d", "AVB_AvalonBay_zscore_60d"], "is_new": false}, {"model_id": "v1_h5_NORMAL_LightGBM_N8", "algo": "LightGBM", "regime": "NORMAL", "horizon": 5, "n_features": 8, "F1_dir": 0.5128, "F1_UP_FORT": 0.2916, "F1_DOWN_FORT": 0.3738, "train_start": "2001-02-07", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["NFCI_ret_5d__minus__NFCI_vol_20d", "heston_var_ev_h1__prod__vix_max_abs_ret_5d", "BTI_BritishAmerican_ret_20d", "heston_var_ev_h1__minus__M_Macys_vol_20d", "Core_CPI_zscore_60d__div__CTAS_Cintas_vol_20d", "EWA_Australia_ret_1d", "AVB_AvalonBay_zscore_60d", "heston_var_ev_h1__div__vix_zscore_10d"], "is_new": false}, {"model_id": "v1_h5_NORMAL_LightGBM_N9", "algo": "LightGBM", "regime": "NORMAL", "horizon": 5, "n_features": 9, "F1_dir": 0.5298, "F1_UP_FORT": 0.3325, "F1_DOWN_FORT": 0.352, "train_start": "2001-02-07", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["NFCI_ret_5d__minus__NFCI_vol_20d", "heston_var_ev_h1__prod__vix_max_abs_ret_5d", "BTI_BritishAmerican_ret_20d", "heston_var_ev_h1__minus__M_Macys_vol_20d", "Core_CPI_zscore_60d__div__CTAS_Cintas_vol_20d", "EWA_Australia_ret_1d", "AVB_AvalonBay_zscore_60d", "heston_var_ev_h1__div__vix_zscore_10d", "HD_zscore_60d"], "is_new": false}, {"model_id": "v1_h5_NORMAL_LightGBM_N10", "algo": "LightGBM", "regime": "NORMAL", "horizon": 5, "n_features": 10, "F1_dir": 0.5098, "F1_UP_FORT": 0.2365, "F1_DOWN_FORT": 0.3317, "train_start": "2001-02-07", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["NFCI_ret_5d__minus__NFCI_vol_20d", "heston_var_ev_h1__prod__vix_max_abs_ret_5d", "BTI_BritishAmerican_ret_20d", "heston_var_ev_h1__minus__M_Macys_vol_20d", "Core_CPI_zscore_60d__div__CTAS_Cintas_vol_20d", "EWA_Australia_ret_1d", "AVB_AvalonBay_zscore_60d", "heston_var_ev_h1__div__vix_zscore_10d", "HD_zscore_60d", "heston_var_ev_h1__div__NFCI_vol_20d"], "is_new": false}, {"model_id": "v1_h5_NORMAL_LightGBM_N12", "algo": "LightGBM", "regime": "NORMAL", "horizon": 5, "n_features": 12, "F1_dir": 0.5107, "F1_UP_FORT": 0.2697, "F1_DOWN_FORT": 0.3467, "train_start": "2001-02-07", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["NFCI_ret_5d__minus__NFCI_vol_20d", "heston_var_ev_h1__prod__vix_max_abs_ret_5d", "BTI_BritishAmerican_ret_20d", "heston_var_ev_h1__minus__M_Macys_vol_20d", "Core_CPI_zscore_60d__div__CTAS_Cintas_vol_20d", "EWA_Australia_ret_1d", "AVB_AvalonBay_zscore_60d", "heston_var_ev_h1__div__vix_zscore_10d", "HD_zscore_60d", "heston_var_ev_h1__div__NFCI_vol_20d", "HUM_Humana_ret_5d", "Core_CPI_zscore_60d__macross__CTAS_Cintas_vol_20d"], "is_new": false}, {"model_id": "v1_h5_NORMAL_LightGBM_N16", "algo": "LightGBM", "regime": "NORMAL", "horizon": 5, "n_features": 16, "F1_dir": 0.5041, "F1_UP_FORT": 0.269, "F1_DOWN_FORT": 0.3392, "train_start": "2001-02-07", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["NFCI_ret_5d__minus__NFCI_vol_20d", "heston_var_ev_h1__prod__vix_max_abs_ret_5d", "BTI_BritishAmerican_ret_20d", "heston_var_ev_h1__minus__M_Macys_vol_20d", "Core_CPI_zscore_60d__div__CTAS_Cintas_vol_20d", "EWA_Australia_ret_1d", "AVB_AvalonBay_zscore_60d", "heston_var_ev_h1__div__vix_zscore_10d", "HD_zscore_60d", "heston_var_ev_h1__div__NFCI_vol_20d", "HUM_Humana_ret_5d", "Core_CPI_zscore_60d__macross__CTAS_Cintas_vol_20d", "NFCI_ret_5d__ret5x__CTAS_Cintas_vol_20d", "XLY_Disc_ret_5d__ret5x__CLX_Clorox_ret_1d", "PG_ret_20d", "NFCI_ret_5d__prod__vix_zscore_10d"], "is_new": false}, {"model_id": "v1_h5_NORMAL_LightGBM_N17", "algo": "LightGBM", "regime": "NORMAL", "horizon": 5, "n_features": 17, "F1_dir": 0.5148, "F1_UP_FORT": 0.2545, "F1_DOWN_FORT": 0.3307, "train_start": "2001-02-07", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["NFCI_ret_5d__minus__NFCI_vol_20d", "heston_var_ev_h1__prod__vix_max_abs_ret_5d", "BTI_BritishAmerican_ret_20d", "heston_var_ev_h1__minus__M_Macys_vol_20d", "Core_CPI_zscore_60d__div__CTAS_Cintas_vol_20d", "EWA_Australia_ret_1d", "AVB_AvalonBay_zscore_60d", "heston_var_ev_h1__div__vix_zscore_10d", "HD_zscore_60d", "heston_var_ev_h1__div__NFCI_vol_20d", "HUM_Humana_ret_5d", "Core_CPI_zscore_60d__macross__CTAS_Cintas_vol_20d", "NFCI_ret_5d__ret5x__CTAS_Cintas_vol_20d", "XLY_Disc_ret_5d__ret5x__CLX_Clorox_ret_1d", "PG_ret_20d", "NFCI_ret_5d__prod__vix_zscore_10d", "Nikkei_Japan_vol_20d"], "is_new": false}, {"model_id": "v1_h5_NORMAL_LightGBM_N18", "algo": "LightGBM", "regime": "NORMAL", "horizon": 5, "n_features": 18, "F1_dir": 0.507, "F1_UP_FORT": 0.2828, "F1_DOWN_FORT": 0.335, "train_start": "2001-02-07", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["NFCI_ret_5d__minus__NFCI_vol_20d", "heston_var_ev_h1__prod__vix_max_abs_ret_5d", "BTI_BritishAmerican_ret_20d", "heston_var_ev_h1__minus__M_Macys_vol_20d", "Core_CPI_zscore_60d__div__CTAS_Cintas_vol_20d", "EWA_Australia_ret_1d", "AVB_AvalonBay_zscore_60d", "heston_var_ev_h1__div__vix_zscore_10d", "HD_zscore_60d", "heston_var_ev_h1__div__NFCI_vol_20d", "HUM_Humana_ret_5d", "Core_CPI_zscore_60d__macross__CTAS_Cintas_vol_20d", "NFCI_ret_5d__ret5x__CTAS_Cintas_vol_20d", "XLY_Disc_ret_5d__ret5x__CLX_Clorox_ret_1d", "PG_ret_20d", "NFCI_ret_5d__prod__vix_zscore_10d", "Nikkei_Japan_vol_20d", "CLX_Clorox_ret_1d__macross__EWH_HongKong_ret_5d"], "is_new": false}, {"model_id": "v1_h5_NORMAL_GradientBoosting_N5", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 5, "n_features": 5, "F1_dir": 0.5174, "F1_UP_FORT": 0.2688, "F1_DOWN_FORT": 0.3047, "train_start": "2001-02-07", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["NFCI_ret_5d__minus__NFCI_vol_20d", "heston_var_ev_h1__prod__vix_max_abs_ret_5d", "BTI_BritishAmerican_ret_20d", "heston_var_ev_h1__minus__M_Macys_vol_20d", "Core_CPI_zscore_60d__div__CTAS_Cintas_vol_20d"], "is_new": false}, {"model_id": "v1_h5_NORMAL_GradientBoosting_N6", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 5, "n_features": 6, "F1_dir": 0.5154, "F1_UP_FORT": 0.3061, "F1_DOWN_FORT": 0.3005, "train_start": "2001-02-07", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["NFCI_ret_5d__minus__NFCI_vol_20d", "heston_var_ev_h1__prod__vix_max_abs_ret_5d", "BTI_BritishAmerican_ret_20d", "heston_var_ev_h1__minus__M_Macys_vol_20d", "Core_CPI_zscore_60d__div__CTAS_Cintas_vol_20d", "EWA_Australia_ret_1d"], "is_new": false}, {"model_id": "v1_h5_NORMAL_GradientBoosting_N7", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 5, "n_features": 7, "F1_dir": 0.5246, "F1_UP_FORT": 0.2929, "F1_DOWN_FORT": 0.3427, "train_start": "2001-02-07", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["NFCI_ret_5d__minus__NFCI_vol_20d", "heston_var_ev_h1__prod__vix_max_abs_ret_5d", "BTI_BritishAmerican_ret_20d", "heston_var_ev_h1__minus__M_Macys_vol_20d", "Core_CPI_zscore_60d__div__CTAS_Cintas_vol_20d", "EWA_Australia_ret_1d", "AVB_AvalonBay_zscore_60d"], "is_new": false}, {"model_id": "v1_h5_NORMAL_GradientBoosting_N8", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 5, "n_features": 8, "F1_dir": 0.5116, "F1_UP_FORT": 0.2952, "F1_DOWN_FORT": 0.3434, "train_start": "2001-02-07", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["NFCI_ret_5d__minus__NFCI_vol_20d", "heston_var_ev_h1__prod__vix_max_abs_ret_5d", "BTI_BritishAmerican_ret_20d", "heston_var_ev_h1__minus__M_Macys_vol_20d", "Core_CPI_zscore_60d__div__CTAS_Cintas_vol_20d", "EWA_Australia_ret_1d", "AVB_AvalonBay_zscore_60d", "heston_var_ev_h1__div__vix_zscore_10d"], "is_new": false}, {"model_id": "v1_h5_NORMAL_GradientBoosting_N9", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 5, "n_features": 9, "F1_dir": 0.5367, "F1_UP_FORT": 0.3096, "F1_DOWN_FORT": 0.3738, "train_start": "2001-02-07", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["NFCI_ret_5d__minus__NFCI_vol_20d", "heston_var_ev_h1__prod__vix_max_abs_ret_5d", "BTI_BritishAmerican_ret_20d", "heston_var_ev_h1__minus__M_Macys_vol_20d", "Core_CPI_zscore_60d__div__CTAS_Cintas_vol_20d", "EWA_Australia_ret_1d", "AVB_AvalonBay_zscore_60d", "heston_var_ev_h1__div__vix_zscore_10d", "HD_zscore_60d"], "is_new": false}, {"model_id": "v1_h5_NORMAL_GradientBoosting_N14", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 5, "n_features": 14, "F1_dir": 0.5283, "F1_UP_FORT": 0.3007, "F1_DOWN_FORT": 0.335, "train_start": "2001-02-07", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["NFCI_ret_5d__minus__NFCI_vol_20d", "heston_var_ev_h1__prod__vix_max_abs_ret_5d", "BTI_BritishAmerican_ret_20d", "heston_var_ev_h1__minus__M_Macys_vol_20d", "Core_CPI_zscore_60d__div__CTAS_Cintas_vol_20d", "EWA_Australia_ret_1d", "AVB_AvalonBay_zscore_60d", "heston_var_ev_h1__div__vix_zscore_10d", "HD_zscore_60d", "heston_var_ev_h1__div__NFCI_vol_20d", "HUM_Humana_ret_5d", "Core_CPI_zscore_60d__macross__CTAS_Cintas_vol_20d", "NFCI_ret_5d__ret5x__CTAS_Cintas_vol_20d", "XLY_Disc_ret_5d__ret5x__CLX_Clorox_ret_1d"], "is_new": false}, {"model_id": "v1_h5_NORMAL_GradientBoosting_N15", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 5, "n_features": 15, "F1_dir": 0.5131, "F1_UP_FORT": 0.302, "F1_DOWN_FORT": 0.325, "train_start": "2001-02-07", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["NFCI_ret_5d__minus__NFCI_vol_20d", "heston_var_ev_h1__prod__vix_max_abs_ret_5d", "BTI_BritishAmerican_ret_20d", "heston_var_ev_h1__minus__M_Macys_vol_20d", "Core_CPI_zscore_60d__div__CTAS_Cintas_vol_20d", "EWA_Australia_ret_1d", "AVB_AvalonBay_zscore_60d", "heston_var_ev_h1__div__vix_zscore_10d", "HD_zscore_60d", "heston_var_ev_h1__div__NFCI_vol_20d", "HUM_Humana_ret_5d", "Core_CPI_zscore_60d__macross__CTAS_Cintas_vol_20d", "NFCI_ret_5d__ret5x__CTAS_Cintas_vol_20d", "XLY_Disc_ret_5d__ret5x__CLX_Clorox_ret_1d", "PG_ret_20d"], "is_new": false}, {"model_id": "v1_h5_NORMAL_GradientBoosting_N16", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 5, "n_features": 16, "F1_dir": 0.5276, "F1_UP_FORT": 0.3054, "F1_DOWN_FORT": 0.3538, "train_start": "2001-02-07", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["NFCI_ret_5d__minus__NFCI_vol_20d", "heston_var_ev_h1__prod__vix_max_abs_ret_5d", "BTI_BritishAmerican_ret_20d", "heston_var_ev_h1__minus__M_Macys_vol_20d", "Core_CPI_zscore_60d__div__CTAS_Cintas_vol_20d", "EWA_Australia_ret_1d", "AVB_AvalonBay_zscore_60d", "heston_var_ev_h1__div__vix_zscore_10d", "HD_zscore_60d", "heston_var_ev_h1__div__NFCI_vol_20d", "HUM_Humana_ret_5d", "Core_CPI_zscore_60d__macross__CTAS_Cintas_vol_20d", "NFCI_ret_5d__ret5x__CTAS_Cintas_vol_20d", "XLY_Disc_ret_5d__ret5x__CLX_Clorox_ret_1d", "PG_ret_20d", "NFCI_ret_5d__prod__vix_zscore_10d"], "is_new": false}, {"model_id": "v1_h5_NORMAL_GradientBoosting_N18", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 5, "n_features": 18, "F1_dir": 0.5024, "F1_UP_FORT": 0.2828, "F1_DOWN_FORT": 0.3161, "train_start": "2001-02-07", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["NFCI_ret_5d__minus__NFCI_vol_20d", "heston_var_ev_h1__prod__vix_max_abs_ret_5d", "BTI_BritishAmerican_ret_20d", "heston_var_ev_h1__minus__M_Macys_vol_20d", "Core_CPI_zscore_60d__div__CTAS_Cintas_vol_20d", "EWA_Australia_ret_1d", "AVB_AvalonBay_zscore_60d", "heston_var_ev_h1__div__vix_zscore_10d", "HD_zscore_60d", "heston_var_ev_h1__div__NFCI_vol_20d", "HUM_Humana_ret_5d", "Core_CPI_zscore_60d__macross__CTAS_Cintas_vol_20d", "NFCI_ret_5d__ret5x__CTAS_Cintas_vol_20d", "XLY_Disc_ret_5d__ret5x__CLX_Clorox_ret_1d", "PG_ret_20d", "NFCI_ret_5d__prod__vix_zscore_10d", "Nikkei_Japan_vol_20d", "CLX_Clorox_ret_1d__macross__EWH_HongKong_ret_5d"], "is_new": false}, {"model_id": "v1_h5_NORMAL_RandomForest_N5", "algo": "RandomForest", "regime": "NORMAL", "horizon": 5, "n_features": 5, "F1_dir": 0.5332, "F1_UP_FORT": 0.2633, "F1_DOWN_FORT": 0.3313, "train_start": "2001-02-07", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["NFCI_ret_5d__minus__NFCI_vol_20d", "heston_var_ev_h1__prod__vix_max_abs_ret_5d", "BTI_BritishAmerican_ret_20d", "heston_var_ev_h1__minus__M_Macys_vol_20d", "Core_CPI_zscore_60d__div__CTAS_Cintas_vol_20d"], "is_new": false}, {"model_id": "v1_h5_NORMAL_RandomForest_N6", "algo": "RandomForest", "regime": "NORMAL", "horizon": 5, "n_features": 6, "F1_dir": 0.5236, "F1_UP_FORT": 0.255, "F1_DOWN_FORT": 0.3615, "train_start": "2001-02-07", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["NFCI_ret_5d__minus__NFCI_vol_20d", "heston_var_ev_h1__prod__vix_max_abs_ret_5d", "BTI_BritishAmerican_ret_20d", "heston_var_ev_h1__minus__M_Macys_vol_20d", "Core_CPI_zscore_60d__div__CTAS_Cintas_vol_20d", "EWA_Australia_ret_1d"], "is_new": false}, {"model_id": "v1_h5_NORMAL_RandomForest_N7", "algo": "RandomForest", "regime": "NORMAL", "horizon": 5, "n_features": 7, "F1_dir": 0.5308, "F1_UP_FORT": 0.2997, "F1_DOWN_FORT": 0.339, "train_start": "2001-02-07", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["NFCI_ret_5d__minus__NFCI_vol_20d", "heston_var_ev_h1__prod__vix_max_abs_ret_5d", "BTI_BritishAmerican_ret_20d", "heston_var_ev_h1__minus__M_Macys_vol_20d", "Core_CPI_zscore_60d__div__CTAS_Cintas_vol_20d", "EWA_Australia_ret_1d", "AVB_AvalonBay_zscore_60d"], "is_new": false}, {"model_id": "v1_h5_NORMAL_RandomForest_N8", "algo": "RandomForest", "regime": "NORMAL", "horizon": 5, "n_features": 8, "F1_dir": 0.5258, "F1_UP_FORT": 0.2636, "F1_DOWN_FORT": 0.3647, "train_start": "2001-02-07", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["NFCI_ret_5d__minus__NFCI_vol_20d", "heston_var_ev_h1__prod__vix_max_abs_ret_5d", "BTI_BritishAmerican_ret_20d", "heston_var_ev_h1__minus__M_Macys_vol_20d", "Core_CPI_zscore_60d__div__CTAS_Cintas_vol_20d", "EWA_Australia_ret_1d", "AVB_AvalonBay_zscore_60d", "heston_var_ev_h1__div__vix_zscore_10d"], "is_new": false}, {"model_id": "v1_h5_NORMAL_RandomForest_N9", "algo": "RandomForest", "regime": "NORMAL", "horizon": 5, "n_features": 9, "F1_dir": 0.524, "F1_UP_FORT": 0.2751, "F1_DOWN_FORT": 0.3619, "train_start": "2001-02-07", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["NFCI_ret_5d__minus__NFCI_vol_20d", "heston_var_ev_h1__prod__vix_max_abs_ret_5d", "BTI_BritishAmerican_ret_20d", "heston_var_ev_h1__minus__M_Macys_vol_20d", "Core_CPI_zscore_60d__div__CTAS_Cintas_vol_20d", "EWA_Australia_ret_1d", "AVB_AvalonBay_zscore_60d", "heston_var_ev_h1__div__vix_zscore_10d", "HD_zscore_60d"], "is_new": false}, {"model_id": "v1_h5_NORMAL_RandomForest_N11", "algo": "RandomForest", "regime": "NORMAL", "horizon": 5, "n_features": 11, "F1_dir": 0.5078, "F1_UP_FORT": 0.2235, "F1_DOWN_FORT": 0.3417, "train_start": "2001-02-07", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["NFCI_ret_5d__minus__NFCI_vol_20d", "heston_var_ev_h1__prod__vix_max_abs_ret_5d", "BTI_BritishAmerican_ret_20d", "heston_var_ev_h1__minus__M_Macys_vol_20d", "Core_CPI_zscore_60d__div__CTAS_Cintas_vol_20d", "EWA_Australia_ret_1d", "AVB_AvalonBay_zscore_60d", "heston_var_ev_h1__div__vix_zscore_10d", "HD_zscore_60d", "heston_var_ev_h1__div__NFCI_vol_20d", "HUM_Humana_ret_5d"], "is_new": false}, {"model_id": "v1_h5_NORMAL_RandomForest_N12", "algo": "RandomForest", "regime": "NORMAL", "horizon": 5, "n_features": 12, "F1_dir": 0.5021, "F1_UP_FORT": 0.2356, "F1_DOWN_FORT": 0.3438, "train_start": "2001-02-07", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["NFCI_ret_5d__minus__NFCI_vol_20d", "heston_var_ev_h1__prod__vix_max_abs_ret_5d", "BTI_BritishAmerican_ret_20d", "heston_var_ev_h1__minus__M_Macys_vol_20d", "Core_CPI_zscore_60d__div__CTAS_Cintas_vol_20d", "EWA_Australia_ret_1d", "AVB_AvalonBay_zscore_60d", "heston_var_ev_h1__div__vix_zscore_10d", "HD_zscore_60d", "heston_var_ev_h1__div__NFCI_vol_20d", "HUM_Humana_ret_5d", "Core_CPI_zscore_60d__macross__CTAS_Cintas_vol_20d"], "is_new": false}, {"model_id": "v1_h5_NORMAL_RandomForest_N13", "algo": "RandomForest", "regime": "NORMAL", "horizon": 5, "n_features": 13, "F1_dir": 0.5055, "F1_UP_FORT": 0.2521, "F1_DOWN_FORT": 0.3485, "train_start": "2001-02-07", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["NFCI_ret_5d__minus__NFCI_vol_20d", "heston_var_ev_h1__prod__vix_max_abs_ret_5d", "BTI_BritishAmerican_ret_20d", "heston_var_ev_h1__minus__M_Macys_vol_20d", "Core_CPI_zscore_60d__div__CTAS_Cintas_vol_20d", "EWA_Australia_ret_1d", "AVB_AvalonBay_zscore_60d", "heston_var_ev_h1__div__vix_zscore_10d", "HD_zscore_60d", "heston_var_ev_h1__div__NFCI_vol_20d", "HUM_Humana_ret_5d", "Core_CPI_zscore_60d__macross__CTAS_Cintas_vol_20d", "NFCI_ret_5d__ret5x__CTAS_Cintas_vol_20d"], "is_new": false}, {"model_id": "v1_h5_NORMAL_RandomForest_N14", "algo": "RandomForest", "regime": "NORMAL", "horizon": 5, "n_features": 14, "F1_dir": 0.5032, "F1_UP_FORT": 0.2438, "F1_DOWN_FORT": 0.3538, "train_start": "2001-02-07", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["NFCI_ret_5d__minus__NFCI_vol_20d", "heston_var_ev_h1__prod__vix_max_abs_ret_5d", "BTI_BritishAmerican_ret_20d", "heston_var_ev_h1__minus__M_Macys_vol_20d", "Core_CPI_zscore_60d__div__CTAS_Cintas_vol_20d", "EWA_Australia_ret_1d", "AVB_AvalonBay_zscore_60d", "heston_var_ev_h1__div__vix_zscore_10d", "HD_zscore_60d", "heston_var_ev_h1__div__NFCI_vol_20d", "HUM_Humana_ret_5d", "Core_CPI_zscore_60d__macross__CTAS_Cintas_vol_20d", "NFCI_ret_5d__ret5x__CTAS_Cintas_vol_20d", "XLY_Disc_ret_5d__ret5x__CLX_Clorox_ret_1d"], "is_new": false}, {"model_id": "v1_h5_NORMAL_RandomForest_N15", "algo": "RandomForest", "regime": "NORMAL", "horizon": 5, "n_features": 15, "F1_dir": 0.5118, "F1_UP_FORT": 0.2678, "F1_DOWN_FORT": 0.3407, "train_start": "2001-02-07", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["NFCI_ret_5d__minus__NFCI_vol_20d", "heston_var_ev_h1__prod__vix_max_abs_ret_5d", "BTI_BritishAmerican_ret_20d", "heston_var_ev_h1__minus__M_Macys_vol_20d", "Core_CPI_zscore_60d__div__CTAS_Cintas_vol_20d", "EWA_Australia_ret_1d", "AVB_AvalonBay_zscore_60d", "heston_var_ev_h1__div__vix_zscore_10d", "HD_zscore_60d", "heston_var_ev_h1__div__NFCI_vol_20d", "HUM_Humana_ret_5d", "Core_CPI_zscore_60d__macross__CTAS_Cintas_vol_20d", "NFCI_ret_5d__ret5x__CTAS_Cintas_vol_20d", "XLY_Disc_ret_5d__ret5x__CLX_Clorox_ret_1d", "PG_ret_20d"], "is_new": false}, {"model_id": "v1_h5_NORMAL_RandomForest_N18", "algo": "RandomForest", "regime": "NORMAL", "horizon": 5, "n_features": 18, "F1_dir": 0.5048, "F1_UP_FORT": 0.2423, "F1_DOWN_FORT": 0.3421, "train_start": "2001-02-07", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["NFCI_ret_5d__minus__NFCI_vol_20d", "heston_var_ev_h1__prod__vix_max_abs_ret_5d", "BTI_BritishAmerican_ret_20d", "heston_var_ev_h1__minus__M_Macys_vol_20d", "Core_CPI_zscore_60d__div__CTAS_Cintas_vol_20d", "EWA_Australia_ret_1d", "AVB_AvalonBay_zscore_60d", "heston_var_ev_h1__div__vix_zscore_10d", "HD_zscore_60d", "heston_var_ev_h1__div__NFCI_vol_20d", "HUM_Humana_ret_5d", "Core_CPI_zscore_60d__macross__CTAS_Cintas_vol_20d", "NFCI_ret_5d__ret5x__CTAS_Cintas_vol_20d", "XLY_Disc_ret_5d__ret5x__CLX_Clorox_ret_1d", "PG_ret_20d", "NFCI_ret_5d__prod__vix_zscore_10d", "Nikkei_Japan_vol_20d", "CLX_Clorox_ret_1d__macross__EWH_HongKong_ret_5d"], "is_new": false}, {"model_id": "v1_h5_NORMAL_LogisticRegression_N6", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 5, "n_features": 6, "F1_dir": 0.5022, "F1_UP_FORT": 0.2446, "F1_DOWN_FORT": 0.3101, "train_start": "2001-02-07", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["NFCI_ret_5d__minus__NFCI_vol_20d", "heston_var_ev_h1__prod__vix_max_abs_ret_5d", "BTI_BritishAmerican_ret_20d", "heston_var_ev_h1__minus__M_Macys_vol_20d", "Core_CPI_zscore_60d__div__CTAS_Cintas_vol_20d", "EWA_Australia_ret_1d"], "is_new": false}, {"model_id": "v1_h5_NORMAL_LogisticRegression_N7", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 5, "n_features": 7, "F1_dir": 0.5172, "F1_UP_FORT": 0.2653, "F1_DOWN_FORT": 0.2933, "train_start": "2001-02-07", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["NFCI_ret_5d__minus__NFCI_vol_20d", "heston_var_ev_h1__prod__vix_max_abs_ret_5d", "BTI_BritishAmerican_ret_20d", "heston_var_ev_h1__minus__M_Macys_vol_20d", "Core_CPI_zscore_60d__div__CTAS_Cintas_vol_20d", "EWA_Australia_ret_1d", "AVB_AvalonBay_zscore_60d"], "is_new": false}, {"model_id": "v1_h5_NORMAL_LogisticRegression_N8", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 5, "n_features": 8, "F1_dir": 0.516, "F1_UP_FORT": 0.2759, "F1_DOWN_FORT": 0.3202, "train_start": "2001-02-07", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["NFCI_ret_5d__minus__NFCI_vol_20d", "heston_var_ev_h1__prod__vix_max_abs_ret_5d", "BTI_BritishAmerican_ret_20d", "heston_var_ev_h1__minus__M_Macys_vol_20d", "Core_CPI_zscore_60d__div__CTAS_Cintas_vol_20d", "EWA_Australia_ret_1d", "AVB_AvalonBay_zscore_60d", "heston_var_ev_h1__div__vix_zscore_10d"], "is_new": false}, {"model_id": "v1_h5_NORMAL_LogisticRegression_N9", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 5, "n_features": 9, "F1_dir": 0.5105, "F1_UP_FORT": 0.2609, "F1_DOWN_FORT": 0.3232, "train_start": "2001-02-07", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["NFCI_ret_5d__minus__NFCI_vol_20d", "heston_var_ev_h1__prod__vix_max_abs_ret_5d", "BTI_BritishAmerican_ret_20d", "heston_var_ev_h1__minus__M_Macys_vol_20d", "Core_CPI_zscore_60d__div__CTAS_Cintas_vol_20d", "EWA_Australia_ret_1d", "AVB_AvalonBay_zscore_60d", "heston_var_ev_h1__div__vix_zscore_10d", "HD_zscore_60d"], "is_new": false}, {"model_id": "v1_h5_NORMAL_LogisticRegression_N17", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 5, "n_features": 17, "F1_dir": 0.5081, "F1_UP_FORT": 0.3042, "F1_DOWN_FORT": 0.3052, "train_start": "2001-02-07", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["NFCI_ret_5d__minus__NFCI_vol_20d", "heston_var_ev_h1__prod__vix_max_abs_ret_5d", "BTI_BritishAmerican_ret_20d", "heston_var_ev_h1__minus__M_Macys_vol_20d", "Core_CPI_zscore_60d__div__CTAS_Cintas_vol_20d", "EWA_Australia_ret_1d", "AVB_AvalonBay_zscore_60d", "heston_var_ev_h1__div__vix_zscore_10d", "HD_zscore_60d", "heston_var_ev_h1__div__NFCI_vol_20d", "HUM_Humana_ret_5d", "Core_CPI_zscore_60d__macross__CTAS_Cintas_vol_20d", "NFCI_ret_5d__ret5x__CTAS_Cintas_vol_20d", "XLY_Disc_ret_5d__ret5x__CLX_Clorox_ret_1d", "PG_ret_20d", "NFCI_ret_5d__prod__vix_zscore_10d", "Nikkei_Japan_vol_20d"], "is_new": false}, {"model_id": "v1_h5_STRESS_XGBoost_N5", "algo": "XGBoost", "regime": "STRESS", "horizon": 5, "n_features": 5, "F1_dir": 0.5437, "F1_UP_FORT": 0.3247, "F1_DOWN_FORT": 0.4686, "train_start": "2000-11-16", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["DE_Deere_vol_20d", "VIX_Price_zscore_60d__prod__PCE_zscore_60d", "Brent_Oil_FRED_ret_20d", "AMD_ret_5d", "SBUX_ret_5d"], "is_new": false}, {"model_id": "v1_h5_STRESS_XGBoost_N6", "algo": "XGBoost", "regime": "STRESS", "horizon": 5, "n_features": 6, "F1_dir": 0.5484, "F1_UP_FORT": 0.3382, "F1_DOWN_FORT": 0.4386, "train_start": "2000-11-16", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["DE_Deere_vol_20d", "VIX_Price_zscore_60d__prod__PCE_zscore_60d", "Brent_Oil_FRED_ret_20d", "AMD_ret_5d", "SBUX_ret_5d", "NFCI_zscore_60d__minus__VIX_Price_zscore_60d"], "is_new": false}, {"model_id": "v1_h5_STRESS_XGBoost_N7", "algo": "XGBoost", "regime": "STRESS", "horizon": 5, "n_features": 7, "F1_dir": 0.5566, "F1_UP_FORT": 0.3383, "F1_DOWN_FORT": 0.4741, "train_start": "2000-11-16", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["DE_Deere_vol_20d", "VIX_Price_zscore_60d__prod__PCE_zscore_60d", "Brent_Oil_FRED_ret_20d", "AMD_ret_5d", "SBUX_ret_5d", "NFCI_zscore_60d__minus__VIX_Price_zscore_60d", "LUV_SouthwestAir_ret_5d"], "is_new": false}, {"model_id": "v1_h5_STRESS_XGBoost_N8", "algo": "XGBoost", "regime": "STRESS", "horizon": 5, "n_features": 8, "F1_dir": 0.5871, "F1_UP_FORT": 0.3768, "F1_DOWN_FORT": 0.5182, "train_start": "2000-11-16", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["DE_Deere_vol_20d", "VIX_Price_zscore_60d__prod__PCE_zscore_60d", "Brent_Oil_FRED_ret_20d", "AMD_ret_5d", "SBUX_ret_5d", "NFCI_zscore_60d__minus__VIX_Price_zscore_60d", "LUV_SouthwestAir_ret_5d", "NFCI_zscore_60d__div__vix_mean_abs_ret_5d"], "is_new": false}, {"model_id": "v1_h5_STRESS_XGBoost_N9", "algo": "XGBoost", "regime": "STRESS", "horizon": 5, "n_features": 9, "F1_dir": 0.5624, "F1_UP_FORT": 0.3902, "F1_DOWN_FORT": 0.51, "train_start": "2000-11-16", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["DE_Deere_vol_20d", "VIX_Price_zscore_60d__prod__PCE_zscore_60d", "Brent_Oil_FRED_ret_20d", "AMD_ret_5d", "SBUX_ret_5d", "NFCI_zscore_60d__minus__VIX_Price_zscore_60d", "LUV_SouthwestAir_ret_5d", "NFCI_zscore_60d__div__vix_mean_abs_ret_5d", "NFCI_zscore_60d__ret5x__SJM_JM_Smucker_ret_5d"], "is_new": false}, {"model_id": "v1_h5_STRESS_XGBoost_N10", "algo": "XGBoost", "regime": "STRESS", "horizon": 5, "n_features": 10, "F1_dir": 0.5625, "F1_UP_FORT": 0.3902, "F1_DOWN_FORT": 0.5122, "train_start": "2000-11-16", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["DE_Deere_vol_20d", "VIX_Price_zscore_60d__prod__PCE_zscore_60d", "Brent_Oil_FRED_ret_20d", "AMD_ret_5d", "SBUX_ret_5d", "NFCI_zscore_60d__minus__VIX_Price_zscore_60d", "LUV_SouthwestAir_ret_5d", "NFCI_zscore_60d__div__vix_mean_abs_ret_5d", "NFCI_zscore_60d__ret5x__SJM_JM_Smucker_ret_5d", "VRTX_VertexPharm_ret_5d__prod__PCE_zscore_60d"], "is_new": false}, {"model_id": "v1_h5_STRESS_LightGBM_N5", "algo": "LightGBM", "regime": "STRESS", "horizon": 5, "n_features": 5, "F1_dir": 0.5372, "F1_UP_FORT": 0.275, "F1_DOWN_FORT": 0.4481, "train_start": "2000-11-16", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["DE_Deere_vol_20d", "VIX_Price_zscore_60d__prod__PCE_zscore_60d", "Brent_Oil_FRED_ret_20d", "AMD_ret_5d", "SBUX_ret_5d"], "is_new": false}, {"model_id": "v1_h5_STRESS_LightGBM_N6", "algo": "LightGBM", "regime": "STRESS", "horizon": 5, "n_features": 6, "F1_dir": 0.5335, "F1_UP_FORT": 0.3455, "F1_DOWN_FORT": 0.4746, "train_start": "2000-11-16", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["DE_Deere_vol_20d", "VIX_Price_zscore_60d__prod__PCE_zscore_60d", "Brent_Oil_FRED_ret_20d", "AMD_ret_5d", "SBUX_ret_5d", "NFCI_zscore_60d__minus__VIX_Price_zscore_60d"], "is_new": false}, {"model_id": "v1_h5_STRESS_LightGBM_N7", "algo": "LightGBM", "regime": "STRESS", "horizon": 5, "n_features": 7, "F1_dir": 0.5275, "F1_UP_FORT": 0.3743, "F1_DOWN_FORT": 0.4667, "train_start": "2000-11-16", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["DE_Deere_vol_20d", "VIX_Price_zscore_60d__prod__PCE_zscore_60d", "Brent_Oil_FRED_ret_20d", "AMD_ret_5d", "SBUX_ret_5d", "NFCI_zscore_60d__minus__VIX_Price_zscore_60d", "LUV_SouthwestAir_ret_5d"], "is_new": false}, {"model_id": "v1_h5_STRESS_LightGBM_N8", "algo": "LightGBM", "regime": "STRESS", "horizon": 5, "n_features": 8, "F1_dir": 0.5399, "F1_UP_FORT": 0.3383, "F1_DOWN_FORT": 0.5122, "train_start": "2000-11-16", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["DE_Deere_vol_20d", "VIX_Price_zscore_60d__prod__PCE_zscore_60d", "Brent_Oil_FRED_ret_20d", "AMD_ret_5d", "SBUX_ret_5d", "NFCI_zscore_60d__minus__VIX_Price_zscore_60d", "LUV_SouthwestAir_ret_5d", "NFCI_zscore_60d__div__vix_mean_abs_ret_5d"], "is_new": false}, {"model_id": "v1_h5_STRESS_LightGBM_N9", "algo": "LightGBM", "regime": "STRESS", "horizon": 5, "n_features": 9, "F1_dir": 0.5387, "F1_UP_FORT": 0.328, "F1_DOWN_FORT": 0.4981, "train_start": "2000-11-16", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["DE_Deere_vol_20d", "VIX_Price_zscore_60d__prod__PCE_zscore_60d", "Brent_Oil_FRED_ret_20d", "AMD_ret_5d", "SBUX_ret_5d", "NFCI_zscore_60d__minus__VIX_Price_zscore_60d", "LUV_SouthwestAir_ret_5d", "NFCI_zscore_60d__div__vix_mean_abs_ret_5d", "NFCI_zscore_60d__ret5x__SJM_JM_Smucker_ret_5d"], "is_new": false}, {"model_id": "v1_h5_STRESS_LightGBM_N10", "algo": "LightGBM", "regime": "STRESS", "horizon": 5, "n_features": 10, "F1_dir": 0.5249, "F1_UP_FORT": 0.3487, "F1_DOWN_FORT": 0.4758, "train_start": "2000-11-16", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["DE_Deere_vol_20d", "VIX_Price_zscore_60d__prod__PCE_zscore_60d", "Brent_Oil_FRED_ret_20d", "AMD_ret_5d", "SBUX_ret_5d", "NFCI_zscore_60d__minus__VIX_Price_zscore_60d", "LUV_SouthwestAir_ret_5d", "NFCI_zscore_60d__div__vix_mean_abs_ret_5d", "NFCI_zscore_60d__ret5x__SJM_JM_Smucker_ret_5d", "VRTX_VertexPharm_ret_5d__prod__PCE_zscore_60d"], "is_new": false}, {"model_id": "v1_h5_STRESS_GradientBoosting_N5", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 5, "n_features": 5, "F1_dir": 0.5578, "F1_UP_FORT": 0.2692, "F1_DOWN_FORT": 0.4661, "train_start": "2000-11-16", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["DE_Deere_vol_20d", "VIX_Price_zscore_60d__prod__PCE_zscore_60d", "Brent_Oil_FRED_ret_20d", "AMD_ret_5d", "SBUX_ret_5d"], "is_new": false}, {"model_id": "v1_h5_STRESS_GradientBoosting_N6", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 5, "n_features": 6, "F1_dir": 0.5504, "F1_UP_FORT": 0.3267, "F1_DOWN_FORT": 0.4163, "train_start": "2000-11-16", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["DE_Deere_vol_20d", "VIX_Price_zscore_60d__prod__PCE_zscore_60d", "Brent_Oil_FRED_ret_20d", "AMD_ret_5d", "SBUX_ret_5d", "NFCI_zscore_60d__minus__VIX_Price_zscore_60d"], "is_new": false}, {"model_id": "v1_h5_STRESS_GradientBoosting_N7", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 5, "n_features": 7, "F1_dir": 0.5335, "F1_UP_FORT": 0.3209, "F1_DOWN_FORT": 0.4474, "train_start": "2000-11-16", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["DE_Deere_vol_20d", "VIX_Price_zscore_60d__prod__PCE_zscore_60d", "Brent_Oil_FRED_ret_20d", "AMD_ret_5d", "SBUX_ret_5d", "NFCI_zscore_60d__minus__VIX_Price_zscore_60d", "LUV_SouthwestAir_ret_5d"], "is_new": false}, {"model_id": "v1_h5_STRESS_GradientBoosting_N8", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 5, "n_features": 8, "F1_dir": 0.5544, "F1_UP_FORT": 0.3627, "F1_DOWN_FORT": 0.4564, "train_start": "2000-11-16", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["DE_Deere_vol_20d", "VIX_Price_zscore_60d__prod__PCE_zscore_60d", "Brent_Oil_FRED_ret_20d", "AMD_ret_5d", "SBUX_ret_5d", "NFCI_zscore_60d__minus__VIX_Price_zscore_60d", "LUV_SouthwestAir_ret_5d", "NFCI_zscore_60d__div__vix_mean_abs_ret_5d"], "is_new": false}, {"model_id": "v1_h5_STRESS_GradientBoosting_N9", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 5, "n_features": 9, "F1_dir": 0.5378, "F1_UP_FORT": 0.3229, "F1_DOWN_FORT": 0.4634, "train_start": "2000-11-16", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["DE_Deere_vol_20d", "VIX_Price_zscore_60d__prod__PCE_zscore_60d", "Brent_Oil_FRED_ret_20d", "AMD_ret_5d", "SBUX_ret_5d", "NFCI_zscore_60d__minus__VIX_Price_zscore_60d", "LUV_SouthwestAir_ret_5d", "NFCI_zscore_60d__div__vix_mean_abs_ret_5d", "NFCI_zscore_60d__ret5x__SJM_JM_Smucker_ret_5d"], "is_new": false}, {"model_id": "v1_h5_STRESS_GradientBoosting_N10", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 5, "n_features": 10, "F1_dir": 0.5389, "F1_UP_FORT": 0.33, "F1_DOWN_FORT": 0.4667, "train_start": "2000-11-16", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["DE_Deere_vol_20d", "VIX_Price_zscore_60d__prod__PCE_zscore_60d", "Brent_Oil_FRED_ret_20d", "AMD_ret_5d", "SBUX_ret_5d", "NFCI_zscore_60d__minus__VIX_Price_zscore_60d", "LUV_SouthwestAir_ret_5d", "NFCI_zscore_60d__div__vix_mean_abs_ret_5d", "NFCI_zscore_60d__ret5x__SJM_JM_Smucker_ret_5d", "VRTX_VertexPharm_ret_5d__prod__PCE_zscore_60d"], "is_new": false}, {"model_id": "v1_h5_STRESS_RandomForest_N5", "algo": "RandomForest", "regime": "STRESS", "horizon": 5, "n_features": 5, "F1_dir": 0.5859, "F1_UP_FORT": 0.3659, "F1_DOWN_FORT": 0.4672, "train_start": "2000-11-16", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["DE_Deere_vol_20d", "VIX_Price_zscore_60d__prod__PCE_zscore_60d", "Brent_Oil_FRED_ret_20d", "AMD_ret_5d", "SBUX_ret_5d"], "is_new": false}, {"model_id": "v1_h5_STRESS_RandomForest_N6", "algo": "RandomForest", "regime": "STRESS", "horizon": 5, "n_features": 6, "F1_dir": 0.5566, "F1_UP_FORT": 0.3691, "F1_DOWN_FORT": 0.5, "train_start": "2000-11-16", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["DE_Deere_vol_20d", "VIX_Price_zscore_60d__prod__PCE_zscore_60d", "Brent_Oil_FRED_ret_20d", "AMD_ret_5d", "SBUX_ret_5d", "NFCI_zscore_60d__minus__VIX_Price_zscore_60d"], "is_new": false}, {"model_id": "v1_h5_STRESS_RandomForest_N7", "algo": "RandomForest", "regime": "STRESS", "horizon": 5, "n_features": 7, "F1_dir": 0.5644, "F1_UP_FORT": 0.3365, "F1_DOWN_FORT": 0.4861, "train_start": "2000-11-16", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["DE_Deere_vol_20d", "VIX_Price_zscore_60d__prod__PCE_zscore_60d", "Brent_Oil_FRED_ret_20d", "AMD_ret_5d", "SBUX_ret_5d", "NFCI_zscore_60d__minus__VIX_Price_zscore_60d", "LUV_SouthwestAir_ret_5d"], "is_new": false}, {"model_id": "v1_h5_STRESS_RandomForest_N8", "algo": "RandomForest", "regime": "STRESS", "horizon": 5, "n_features": 8, "F1_dir": 0.582, "F1_UP_FORT": 0.3843, "F1_DOWN_FORT": 0.5077, "train_start": "2000-11-16", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["DE_Deere_vol_20d", "VIX_Price_zscore_60d__prod__PCE_zscore_60d", "Brent_Oil_FRED_ret_20d", "AMD_ret_5d", "SBUX_ret_5d", "NFCI_zscore_60d__minus__VIX_Price_zscore_60d", "LUV_SouthwestAir_ret_5d", "NFCI_zscore_60d__div__vix_mean_abs_ret_5d"], "is_new": false}, {"model_id": "v1_h5_STRESS_RandomForest_N9", "algo": "RandomForest", "regime": "STRESS", "horizon": 5, "n_features": 9, "F1_dir": 0.5739, "F1_UP_FORT": 0.3767, "F1_DOWN_FORT": 0.5483, "train_start": "2000-11-16", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["DE_Deere_vol_20d", "VIX_Price_zscore_60d__prod__PCE_zscore_60d", "Brent_Oil_FRED_ret_20d", "AMD_ret_5d", "SBUX_ret_5d", "NFCI_zscore_60d__minus__VIX_Price_zscore_60d", "LUV_SouthwestAir_ret_5d", "NFCI_zscore_60d__div__vix_mean_abs_ret_5d", "NFCI_zscore_60d__ret5x__SJM_JM_Smucker_ret_5d"], "is_new": false}, {"model_id": "v1_h5_STRESS_RandomForest_N10", "algo": "RandomForest", "regime": "STRESS", "horizon": 5, "n_features": 10, "F1_dir": 0.5735, "F1_UP_FORT": 0.3849, "F1_DOWN_FORT": 0.5476, "train_start": "2000-11-16", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["DE_Deere_vol_20d", "VIX_Price_zscore_60d__prod__PCE_zscore_60d", "Brent_Oil_FRED_ret_20d", "AMD_ret_5d", "SBUX_ret_5d", "NFCI_zscore_60d__minus__VIX_Price_zscore_60d", "LUV_SouthwestAir_ret_5d", "NFCI_zscore_60d__div__vix_mean_abs_ret_5d", "NFCI_zscore_60d__ret5x__SJM_JM_Smucker_ret_5d", "VRTX_VertexPharm_ret_5d__prod__PCE_zscore_60d"], "is_new": false}, {"model_id": "v1_h5_STRESS_LogisticRegression_N5", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 5, "n_features": 5, "F1_dir": 0.5747, "F1_UP_FORT": 0.256, "F1_DOWN_FORT": 0.5, "train_start": "2000-11-16", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["DE_Deere_vol_20d", "VIX_Price_zscore_60d__prod__PCE_zscore_60d", "Brent_Oil_FRED_ret_20d", "AMD_ret_5d", "SBUX_ret_5d"], "is_new": false}, {"model_id": "v1_h5_STRESS_XGBoost_Optuna_N8", "algo": "XGBoost", "regime": "STRESS", "horizon": 5, "n_features": 8, "F1_dir": 0.5618, "F1_UP_FORT": 0.356, "F1_DOWN_FORT": 0.4686, "train_start": "2000-11-16", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["DE_Deere_vol_20d", "VIX_Price_zscore_60d__prod__PCE_zscore_60d", "Brent_Oil_FRED_ret_20d", "AMD_ret_5d", "SBUX_ret_5d", "NFCI_zscore_60d__minus__VIX_Price_zscore_60d", "LUV_SouthwestAir_ret_5d", "NFCI_zscore_60d__div__vix_mean_abs_ret_5d"], "is_new": false}, {"model_id": "v1_h5_STRESS_XGBoost_OptunaCal_N8", "algo": "XGBoostCal", "regime": "STRESS", "horizon": 5, "n_features": 8, "F1_dir": 0.5618, "F1_UP_FORT": 0.356, "F1_DOWN_FORT": 0.4686, "train_start": "2000-11-16", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["DE_Deere_vol_20d", "VIX_Price_zscore_60d__prod__PCE_zscore_60d", "Brent_Oil_FRED_ret_20d", "AMD_ret_5d", "SBUX_ret_5d", "NFCI_zscore_60d__minus__VIX_Price_zscore_60d", "LUV_SouthwestAir_ret_5d", "NFCI_zscore_60d__div__vix_mean_abs_ret_5d"], "is_new": false}, {"model_id": "v1_h5_GLOBAL_XGBoost_N5", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 5, "n_features": 5, "F1_dir": 0.5781, "F1_UP_FORT": 0.3818, "F1_DOWN_FORT": 0.3668, "train_start": "2001-02-07", "sampler": "SMOTEENN", "best_params": "{}", "features": ["NFCI_ret_5d__div__vix_vol_of_vol_10d", "SJM_JM_Smucker_ret_5d", "NFCI_ret_5d__minus__vix_level", "vix_vol_of_vol_10d__prod__NFCI_zscore_60d", "NFCI_vol_20d__ret5x__HD_zscore_60d"], "is_new": false}, {"model_id": "v1_h5_GLOBAL_XGBoost_N6", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 5, "n_features": 6, "F1_dir": 0.5794, "F1_UP_FORT": 0.4035, "F1_DOWN_FORT": 0.3594, "train_start": "2001-02-07", "sampler": "SMOTEENN", "best_params": "{}", "features": ["NFCI_ret_5d__div__vix_vol_of_vol_10d", "SJM_JM_Smucker_ret_5d", "NFCI_ret_5d__minus__vix_level", "vix_vol_of_vol_10d__prod__NFCI_zscore_60d", "NFCI_vol_20d__ret5x__HD_zscore_60d", "NFCI_ret_5d__div__NFCI_vol_20d"], "is_new": false}, {"model_id": "v1_h5_GLOBAL_XGBoost_N7", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 5, "n_features": 7, "F1_dir": 0.5983, "F1_UP_FORT": 0.3904, "F1_DOWN_FORT": 0.4586, "train_start": "2001-02-07", "sampler": "SMOTEENN", "best_params": "{}", "features": ["NFCI_ret_5d__div__vix_vol_of_vol_10d", "SJM_JM_Smucker_ret_5d", "NFCI_ret_5d__minus__vix_level", "vix_vol_of_vol_10d__prod__NFCI_zscore_60d", "NFCI_vol_20d__ret5x__HD_zscore_60d", "NFCI_ret_5d__div__NFCI_vol_20d", "vix_zscore_10d__zrel__HON_Honeywell_zscore_60d"], "is_new": false}, {"model_id": "v1_h5_GLOBAL_XGBoost_N8", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 5, "n_features": 8, "F1_dir": 0.6195, "F1_UP_FORT": 0.4124, "F1_DOWN_FORT": 0.4619, "train_start": "2001-02-07", "sampler": "SMOTEENN", "best_params": "{}", "features": ["NFCI_ret_5d__div__vix_vol_of_vol_10d", "SJM_JM_Smucker_ret_5d", "NFCI_ret_5d__minus__vix_level", "vix_vol_of_vol_10d__prod__NFCI_zscore_60d", "NFCI_vol_20d__ret5x__HD_zscore_60d", "NFCI_ret_5d__div__NFCI_vol_20d", "vix_zscore_10d__zrel__HON_Honeywell_zscore_60d", "NFCI_zscore_60d__minus__vix_ma_20"], "is_new": false}, {"model_id": "v1_h5_GLOBAL_XGBoost_N9", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 5, "n_features": 9, "F1_dir": 0.5883, "F1_UP_FORT": 0.3827, "F1_DOWN_FORT": 0.4519, "train_start": "2001-02-07", "sampler": "SMOTEENN", "best_params": "{}", "features": ["NFCI_ret_5d__div__vix_vol_of_vol_10d", "SJM_JM_Smucker_ret_5d", "NFCI_ret_5d__minus__vix_level", "vix_vol_of_vol_10d__prod__NFCI_zscore_60d", "NFCI_vol_20d__ret5x__HD_zscore_60d", "NFCI_ret_5d__div__NFCI_vol_20d", "vix_zscore_10d__zrel__HON_Honeywell_zscore_60d", "NFCI_zscore_60d__minus__vix_ma_20", "heston_ev_h3"], "is_new": false}, {"model_id": "v1_h5_GLOBAL_XGBoost_N10", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 5, "n_features": 10, "F1_dir": 0.5725, "F1_UP_FORT": 0.3638, "F1_DOWN_FORT": 0.4597, "train_start": "2001-02-07", "sampler": "SMOTEENN", "best_params": "{}", "features": ["NFCI_ret_5d__div__vix_vol_of_vol_10d", "SJM_JM_Smucker_ret_5d", "NFCI_ret_5d__minus__vix_level", "vix_vol_of_vol_10d__prod__NFCI_zscore_60d", "NFCI_vol_20d__ret5x__HD_zscore_60d", "NFCI_ret_5d__div__NFCI_vol_20d", "vix_zscore_10d__zrel__HON_Honeywell_zscore_60d", "NFCI_zscore_60d__minus__vix_ma_20", "heston_ev_h3", "TED_Spread_vol_20d"], "is_new": false}, {"model_id": "v1_h5_GLOBAL_XGBoost_N11", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 5, "n_features": 11, "F1_dir": 0.5843, "F1_UP_FORT": 0.3761, "F1_DOWN_FORT": 0.4671, "train_start": "2001-02-07", "sampler": "SMOTEENN", "best_params": "{}", "features": ["NFCI_ret_5d__div__vix_vol_of_vol_10d", "SJM_JM_Smucker_ret_5d", "NFCI_ret_5d__minus__vix_level", "vix_vol_of_vol_10d__prod__NFCI_zscore_60d", "NFCI_vol_20d__ret5x__HD_zscore_60d", "NFCI_ret_5d__div__NFCI_vol_20d", "vix_zscore_10d__zrel__HON_Honeywell_zscore_60d", "NFCI_zscore_60d__minus__vix_ma_20", "heston_ev_h3", "TED_Spread_vol_20d", "NFCI_vol_20d__ret5x__VVIX_vol_20d"], "is_new": false}, {"model_id": "v1_h5_GLOBAL_XGBoost_N12", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 5, "n_features": 12, "F1_dir": 0.5757, "F1_UP_FORT": 0.3676, "F1_DOWN_FORT": 0.4635, "train_start": "2001-02-07", "sampler": "SMOTEENN", "best_params": "{}", "features": ["NFCI_ret_5d__div__vix_vol_of_vol_10d", "SJM_JM_Smucker_ret_5d", "NFCI_ret_5d__minus__vix_level", "vix_vol_of_vol_10d__prod__NFCI_zscore_60d", "NFCI_vol_20d__ret5x__HD_zscore_60d", "NFCI_ret_5d__div__NFCI_vol_20d", "vix_zscore_10d__zrel__HON_Honeywell_zscore_60d", "NFCI_zscore_60d__minus__vix_ma_20", "heston_ev_h3", "TED_Spread_vol_20d", "NFCI_vol_20d__ret5x__VVIX_vol_20d", "NFCI_ret_5d__div__vix_level"], "is_new": false}, {"model_id": "v1_h5_GLOBAL_XGBoost_N13", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 5, "n_features": 13, "F1_dir": 0.5887, "F1_UP_FORT": 0.3709, "F1_DOWN_FORT": 0.4591, "train_start": "2001-02-07", "sampler": "SMOTEENN", "best_params": "{}", "features": ["NFCI_ret_5d__div__vix_vol_of_vol_10d", "SJM_JM_Smucker_ret_5d", "NFCI_ret_5d__minus__vix_level", "vix_vol_of_vol_10d__prod__NFCI_zscore_60d", "NFCI_vol_20d__ret5x__HD_zscore_60d", "NFCI_ret_5d__div__NFCI_vol_20d", "vix_zscore_10d__zrel__HON_Honeywell_zscore_60d", "NFCI_zscore_60d__minus__vix_ma_20", "heston_ev_h3", "TED_Spread_vol_20d", "NFCI_vol_20d__ret5x__VVIX_vol_20d", "NFCI_ret_5d__div__vix_level", "US30Y_Rate_ret_20d"], "is_new": false}, {"model_id": "v1_h5_GLOBAL_XGBoost_N14", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 5, "n_features": 14, "F1_dir": 0.6057, "F1_UP_FORT": 0.3958, "F1_DOWN_FORT": 0.4487, "train_start": "2001-02-07", "sampler": "SMOTEENN", "best_params": "{}", "features": ["NFCI_ret_5d__div__vix_vol_of_vol_10d", "SJM_JM_Smucker_ret_5d", "NFCI_ret_5d__minus__vix_level", "vix_vol_of_vol_10d__prod__NFCI_zscore_60d", "NFCI_vol_20d__ret5x__HD_zscore_60d", "NFCI_ret_5d__div__NFCI_vol_20d", "vix_zscore_10d__zrel__HON_Honeywell_zscore_60d", "NFCI_zscore_60d__minus__vix_ma_20", "heston_ev_h3", "TED_Spread_vol_20d", "NFCI_vol_20d__ret5x__VVIX_vol_20d", "NFCI_ret_5d__div__vix_level", "US30Y_Rate_ret_20d", "EWM_Malaysia_zscore_60d"], "is_new": false}, {"model_id": "v1_h5_GLOBAL_XGBoost_N15", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 5, "n_features": 15, "F1_dir": 0.597, "F1_UP_FORT": 0.3907, "F1_DOWN_FORT": 0.4523, "train_start": "2001-02-07", "sampler": "SMOTEENN", "best_params": "{}", "features": ["NFCI_ret_5d__div__vix_vol_of_vol_10d", "SJM_JM_Smucker_ret_5d", "NFCI_ret_5d__minus__vix_level", "vix_vol_of_vol_10d__prod__NFCI_zscore_60d", "NFCI_vol_20d__ret5x__HD_zscore_60d", "NFCI_ret_5d__div__NFCI_vol_20d", "vix_zscore_10d__zrel__HON_Honeywell_zscore_60d", "NFCI_zscore_60d__minus__vix_ma_20", "heston_ev_h3", "TED_Spread_vol_20d", "NFCI_vol_20d__ret5x__VVIX_vol_20d", "NFCI_ret_5d__div__vix_level", "US30Y_Rate_ret_20d", "EWM_Malaysia_zscore_60d", "PAYX_Paychex_vol_20d"], "is_new": false}, {"model_id": "v1_h5_GLOBAL_LightGBM_N5", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 5, "n_features": 5, "F1_dir": 0.56, "F1_UP_FORT": 0.3625, "F1_DOWN_FORT": 0.3701, "train_start": "2001-02-07", "sampler": "SMOTEENN", "best_params": "{}", "features": ["NFCI_ret_5d__div__vix_vol_of_vol_10d", "SJM_JM_Smucker_ret_5d", "NFCI_ret_5d__minus__vix_level", "vix_vol_of_vol_10d__prod__NFCI_zscore_60d", "NFCI_vol_20d__ret5x__HD_zscore_60d"], "is_new": false}, {"model_id": "v1_h5_GLOBAL_LightGBM_N6", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 5, "n_features": 6, "F1_dir": 0.5608, "F1_UP_FORT": 0.3548, "F1_DOWN_FORT": 0.3255, "train_start": "2001-02-07", "sampler": "SMOTEENN", "best_params": "{}", "features": ["NFCI_ret_5d__div__vix_vol_of_vol_10d", "SJM_JM_Smucker_ret_5d", "NFCI_ret_5d__minus__vix_level", "vix_vol_of_vol_10d__prod__NFCI_zscore_60d", "NFCI_vol_20d__ret5x__HD_zscore_60d", "NFCI_ret_5d__div__NFCI_vol_20d"], "is_new": false}, {"model_id": "v1_h5_GLOBAL_LightGBM_N7", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 5, "n_features": 7, "F1_dir": 0.5926, "F1_UP_FORT": 0.3707, "F1_DOWN_FORT": 0.4211, "train_start": "2001-02-07", "sampler": "SMOTEENN", "best_params": "{}", "features": ["NFCI_ret_5d__div__vix_vol_of_vol_10d", "SJM_JM_Smucker_ret_5d", "NFCI_ret_5d__minus__vix_level", "vix_vol_of_vol_10d__prod__NFCI_zscore_60d", "NFCI_vol_20d__ret5x__HD_zscore_60d", "NFCI_ret_5d__div__NFCI_vol_20d", "vix_zscore_10d__zrel__HON_Honeywell_zscore_60d"], "is_new": false}, {"model_id": "v1_h5_GLOBAL_LightGBM_N8", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 5, "n_features": 8, "F1_dir": 0.6081, "F1_UP_FORT": 0.3708, "F1_DOWN_FORT": 0.4174, "train_start": "2001-02-07", "sampler": "SMOTEENN", "best_params": "{}", "features": ["NFCI_ret_5d__div__vix_vol_of_vol_10d", "SJM_JM_Smucker_ret_5d", "NFCI_ret_5d__minus__vix_level", "vix_vol_of_vol_10d__prod__NFCI_zscore_60d", "NFCI_vol_20d__ret5x__HD_zscore_60d", "NFCI_ret_5d__div__NFCI_vol_20d", "vix_zscore_10d__zrel__HON_Honeywell_zscore_60d", "NFCI_zscore_60d__minus__vix_ma_20"], "is_new": false}, {"model_id": "v1_h5_GLOBAL_LightGBM_N9", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 5, "n_features": 9, "F1_dir": 0.5808, "F1_UP_FORT": 0.3643, "F1_DOWN_FORT": 0.4224, "train_start": "2001-02-07", "sampler": "SMOTEENN", "best_params": "{}", "features": ["NFCI_ret_5d__div__vix_vol_of_vol_10d", "SJM_JM_Smucker_ret_5d", "NFCI_ret_5d__minus__vix_level", "vix_vol_of_vol_10d__prod__NFCI_zscore_60d", "NFCI_vol_20d__ret5x__HD_zscore_60d", "NFCI_ret_5d__div__NFCI_vol_20d", "vix_zscore_10d__zrel__HON_Honeywell_zscore_60d", "NFCI_zscore_60d__minus__vix_ma_20", "heston_ev_h3"], "is_new": false}, {"model_id": "v1_h5_GLOBAL_LightGBM_N10", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 5, "n_features": 10, "F1_dir": 0.5898, "F1_UP_FORT": 0.3418, "F1_DOWN_FORT": 0.4379, "train_start": "2001-02-07", "sampler": "SMOTEENN", "best_params": "{}", "features": ["NFCI_ret_5d__div__vix_vol_of_vol_10d", "SJM_JM_Smucker_ret_5d", "NFCI_ret_5d__minus__vix_level", "vix_vol_of_vol_10d__prod__NFCI_zscore_60d", "NFCI_vol_20d__ret5x__HD_zscore_60d", "NFCI_ret_5d__div__NFCI_vol_20d", "vix_zscore_10d__zrel__HON_Honeywell_zscore_60d", "NFCI_zscore_60d__minus__vix_ma_20", "heston_ev_h3", "TED_Spread_vol_20d"], "is_new": false}, {"model_id": "v1_h5_GLOBAL_LightGBM_N11", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 5, "n_features": 11, "F1_dir": 0.5816, "F1_UP_FORT": 0.3438, "F1_DOWN_FORT": 0.4382, "train_start": "2001-02-07", "sampler": "SMOTEENN", "best_params": "{}", "features": ["NFCI_ret_5d__div__vix_vol_of_vol_10d", "SJM_JM_Smucker_ret_5d", "NFCI_ret_5d__minus__vix_level", "vix_vol_of_vol_10d__prod__NFCI_zscore_60d", "NFCI_vol_20d__ret5x__HD_zscore_60d", "NFCI_ret_5d__div__NFCI_vol_20d", "vix_zscore_10d__zrel__HON_Honeywell_zscore_60d", "NFCI_zscore_60d__minus__vix_ma_20", "heston_ev_h3", "TED_Spread_vol_20d", "NFCI_vol_20d__ret5x__VVIX_vol_20d"], "is_new": false}, {"model_id": "v1_h5_GLOBAL_LightGBM_N12", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 5, "n_features": 12, "F1_dir": 0.5818, "F1_UP_FORT": 0.3571, "F1_DOWN_FORT": 0.431, "train_start": "2001-02-07", "sampler": "SMOTEENN", "best_params": "{}", "features": ["NFCI_ret_5d__div__vix_vol_of_vol_10d", "SJM_JM_Smucker_ret_5d", "NFCI_ret_5d__minus__vix_level", "vix_vol_of_vol_10d__prod__NFCI_zscore_60d", "NFCI_vol_20d__ret5x__HD_zscore_60d", "NFCI_ret_5d__div__NFCI_vol_20d", "vix_zscore_10d__zrel__HON_Honeywell_zscore_60d", "NFCI_zscore_60d__minus__vix_ma_20", "heston_ev_h3", "TED_Spread_vol_20d", "NFCI_vol_20d__ret5x__VVIX_vol_20d", "NFCI_ret_5d__div__vix_level"], "is_new": false}, {"model_id": "v1_h5_GLOBAL_LightGBM_N13", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 5, "n_features": 13, "F1_dir": 0.5864, "F1_UP_FORT": 0.3485, "F1_DOWN_FORT": 0.4324, "train_start": "2001-02-07", "sampler": "SMOTEENN", "best_params": "{}", "features": ["NFCI_ret_5d__div__vix_vol_of_vol_10d", "SJM_JM_Smucker_ret_5d", "NFCI_ret_5d__minus__vix_level", "vix_vol_of_vol_10d__prod__NFCI_zscore_60d", "NFCI_vol_20d__ret5x__HD_zscore_60d", "NFCI_ret_5d__div__NFCI_vol_20d", "vix_zscore_10d__zrel__HON_Honeywell_zscore_60d", "NFCI_zscore_60d__minus__vix_ma_20", "heston_ev_h3", "TED_Spread_vol_20d", "NFCI_vol_20d__ret5x__VVIX_vol_20d", "NFCI_ret_5d__div__vix_level", "US30Y_Rate_ret_20d"], "is_new": false}, {"model_id": "v1_h5_GLOBAL_LightGBM_N14", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 5, "n_features": 14, "F1_dir": 0.5978, "F1_UP_FORT": 0.3777, "F1_DOWN_FORT": 0.4379, "train_start": "2001-02-07", "sampler": "SMOTEENN", "best_params": "{}", "features": ["NFCI_ret_5d__div__vix_vol_of_vol_10d", "SJM_JM_Smucker_ret_5d", "NFCI_ret_5d__minus__vix_level", "vix_vol_of_vol_10d__prod__NFCI_zscore_60d", "NFCI_vol_20d__ret5x__HD_zscore_60d", "NFCI_ret_5d__div__NFCI_vol_20d", "vix_zscore_10d__zrel__HON_Honeywell_zscore_60d", "NFCI_zscore_60d__minus__vix_ma_20", "heston_ev_h3", "TED_Spread_vol_20d", "NFCI_vol_20d__ret5x__VVIX_vol_20d", "NFCI_ret_5d__div__vix_level", "US30Y_Rate_ret_20d", "EWM_Malaysia_zscore_60d"], "is_new": false}, {"model_id": "v1_h5_GLOBAL_LightGBM_N15", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 5, "n_features": 15, "F1_dir": 0.6043, "F1_UP_FORT": 0.3831, "F1_DOWN_FORT": 0.4399, "train_start": "2001-02-07", "sampler": "SMOTEENN", "best_params": "{}", "features": ["NFCI_ret_5d__div__vix_vol_of_vol_10d", "SJM_JM_Smucker_ret_5d", "NFCI_ret_5d__minus__vix_level", "vix_vol_of_vol_10d__prod__NFCI_zscore_60d", "NFCI_vol_20d__ret5x__HD_zscore_60d", "NFCI_ret_5d__div__NFCI_vol_20d", "vix_zscore_10d__zrel__HON_Honeywell_zscore_60d", "NFCI_zscore_60d__minus__vix_ma_20", "heston_ev_h3", "TED_Spread_vol_20d", "NFCI_vol_20d__ret5x__VVIX_vol_20d", "NFCI_ret_5d__div__vix_level", "US30Y_Rate_ret_20d", "EWM_Malaysia_zscore_60d", "PAYX_Paychex_vol_20d"], "is_new": false}, {"model_id": "v1_h5_GLOBAL_GradientBoosting_N5", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 5, "n_features": 5, "F1_dir": 0.561, "F1_UP_FORT": 0.3927, "F1_DOWN_FORT": 0.3555, "train_start": "2001-02-07", "sampler": "SMOTEENN", "best_params": "{}", "features": ["NFCI_ret_5d__div__vix_vol_of_vol_10d", "SJM_JM_Smucker_ret_5d", "NFCI_ret_5d__minus__vix_level", "vix_vol_of_vol_10d__prod__NFCI_zscore_60d", "NFCI_vol_20d__ret5x__HD_zscore_60d"], "is_new": false}, {"model_id": "v1_h5_GLOBAL_GradientBoosting_N6", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 5, "n_features": 6, "F1_dir": 0.5475, "F1_UP_FORT": 0.3701, "F1_DOWN_FORT": 0.3645, "train_start": "2001-02-07", "sampler": "SMOTEENN", "best_params": "{}", "features": ["NFCI_ret_5d__div__vix_vol_of_vol_10d", "SJM_JM_Smucker_ret_5d", "NFCI_ret_5d__minus__vix_level", "vix_vol_of_vol_10d__prod__NFCI_zscore_60d", "NFCI_vol_20d__ret5x__HD_zscore_60d", "NFCI_ret_5d__div__NFCI_vol_20d"], "is_new": false}, {"model_id": "v1_h5_GLOBAL_GradientBoosting_N7", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 5, "n_features": 7, "F1_dir": 0.5927, "F1_UP_FORT": 0.3892, "F1_DOWN_FORT": 0.4301, "train_start": "2001-02-07", "sampler": "SMOTEENN", "best_params": "{}", "features": ["NFCI_ret_5d__div__vix_vol_of_vol_10d", "SJM_JM_Smucker_ret_5d", "NFCI_ret_5d__minus__vix_level", "vix_vol_of_vol_10d__prod__NFCI_zscore_60d", "NFCI_vol_20d__ret5x__HD_zscore_60d", "NFCI_ret_5d__div__NFCI_vol_20d", "vix_zscore_10d__zrel__HON_Honeywell_zscore_60d"], "is_new": false}, {"model_id": "v1_h5_GLOBAL_GradientBoosting_N8", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 5, "n_features": 8, "F1_dir": 0.587, "F1_UP_FORT": 0.4026, "F1_DOWN_FORT": 0.4296, "train_start": "2001-02-07", "sampler": "SMOTEENN", "best_params": "{}", "features": ["NFCI_ret_5d__div__vix_vol_of_vol_10d", "SJM_JM_Smucker_ret_5d", "NFCI_ret_5d__minus__vix_level", "vix_vol_of_vol_10d__prod__NFCI_zscore_60d", "NFCI_vol_20d__ret5x__HD_zscore_60d", "NFCI_ret_5d__div__NFCI_vol_20d", "vix_zscore_10d__zrel__HON_Honeywell_zscore_60d", "NFCI_zscore_60d__minus__vix_ma_20"], "is_new": false}, {"model_id": "v1_h5_GLOBAL_GradientBoosting_N9", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 5, "n_features": 9, "F1_dir": 0.5819, "F1_UP_FORT": 0.3749, "F1_DOWN_FORT": 0.4379, "train_start": "2001-02-07", "sampler": "SMOTEENN", "best_params": "{}", "features": ["NFCI_ret_5d__div__vix_vol_of_vol_10d", "SJM_JM_Smucker_ret_5d", "NFCI_ret_5d__minus__vix_level", "vix_vol_of_vol_10d__prod__NFCI_zscore_60d", "NFCI_vol_20d__ret5x__HD_zscore_60d", "NFCI_ret_5d__div__NFCI_vol_20d", "vix_zscore_10d__zrel__HON_Honeywell_zscore_60d", "NFCI_zscore_60d__minus__vix_ma_20", "heston_ev_h3"], "is_new": false}, {"model_id": "v1_h5_GLOBAL_GradientBoosting_N10", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 5, "n_features": 10, "F1_dir": 0.5587, "F1_UP_FORT": 0.3256, "F1_DOWN_FORT": 0.4351, "train_start": "2001-02-07", "sampler": "SMOTEENN", "best_params": "{}", "features": ["NFCI_ret_5d__div__vix_vol_of_vol_10d", "SJM_JM_Smucker_ret_5d", "NFCI_ret_5d__minus__vix_level", "vix_vol_of_vol_10d__prod__NFCI_zscore_60d", "NFCI_vol_20d__ret5x__HD_zscore_60d", "NFCI_ret_5d__div__NFCI_vol_20d", "vix_zscore_10d__zrel__HON_Honeywell_zscore_60d", "NFCI_zscore_60d__minus__vix_ma_20", "heston_ev_h3", "TED_Spread_vol_20d"], "is_new": false}, {"model_id": "v1_h5_GLOBAL_GradientBoosting_N11", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 5, "n_features": 11, "F1_dir": 0.5575, "F1_UP_FORT": 0.3346, "F1_DOWN_FORT": 0.4404, "train_start": "2001-02-07", "sampler": "SMOTEENN", "best_params": "{}", "features": ["NFCI_ret_5d__div__vix_vol_of_vol_10d", "SJM_JM_Smucker_ret_5d", "NFCI_ret_5d__minus__vix_level", "vix_vol_of_vol_10d__prod__NFCI_zscore_60d", "NFCI_vol_20d__ret5x__HD_zscore_60d", "NFCI_ret_5d__div__NFCI_vol_20d", "vix_zscore_10d__zrel__HON_Honeywell_zscore_60d", "NFCI_zscore_60d__minus__vix_ma_20", "heston_ev_h3", "TED_Spread_vol_20d", "NFCI_vol_20d__ret5x__VVIX_vol_20d"], "is_new": false}, {"model_id": "v1_h5_GLOBAL_GradientBoosting_N12", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 5, "n_features": 12, "F1_dir": 0.5618, "F1_UP_FORT": 0.3453, "F1_DOWN_FORT": 0.4372, "train_start": "2001-02-07", "sampler": "SMOTEENN", "best_params": "{}", "features": ["NFCI_ret_5d__div__vix_vol_of_vol_10d", "SJM_JM_Smucker_ret_5d", "NFCI_ret_5d__minus__vix_level", "vix_vol_of_vol_10d__prod__NFCI_zscore_60d", "NFCI_vol_20d__ret5x__HD_zscore_60d", "NFCI_ret_5d__div__NFCI_vol_20d", "vix_zscore_10d__zrel__HON_Honeywell_zscore_60d", "NFCI_zscore_60d__minus__vix_ma_20", "heston_ev_h3", "TED_Spread_vol_20d", "NFCI_vol_20d__ret5x__VVIX_vol_20d", "NFCI_ret_5d__div__vix_level"], "is_new": false}, {"model_id": "v1_h5_GLOBAL_GradientBoosting_N13", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 5, "n_features": 13, "F1_dir": 0.58, "F1_UP_FORT": 0.3535, "F1_DOWN_FORT": 0.4417, "train_start": "2001-02-07", "sampler": "SMOTEENN", "best_params": "{}", "features": ["NFCI_ret_5d__div__vix_vol_of_vol_10d", "SJM_JM_Smucker_ret_5d", "NFCI_ret_5d__minus__vix_level", "vix_vol_of_vol_10d__prod__NFCI_zscore_60d", "NFCI_vol_20d__ret5x__HD_zscore_60d", "NFCI_ret_5d__div__NFCI_vol_20d", "vix_zscore_10d__zrel__HON_Honeywell_zscore_60d", "NFCI_zscore_60d__minus__vix_ma_20", "heston_ev_h3", "TED_Spread_vol_20d", "NFCI_vol_20d__ret5x__VVIX_vol_20d", "NFCI_ret_5d__div__vix_level", "US30Y_Rate_ret_20d"], "is_new": false}, {"model_id": "v1_h5_GLOBAL_GradientBoosting_N14", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 5, "n_features": 14, "F1_dir": 0.5845, "F1_UP_FORT": 0.3711, "F1_DOWN_FORT": 0.4453, "train_start": "2001-02-07", "sampler": "SMOTEENN", "best_params": "{}", "features": ["NFCI_ret_5d__div__vix_vol_of_vol_10d", "SJM_JM_Smucker_ret_5d", "NFCI_ret_5d__minus__vix_level", "vix_vol_of_vol_10d__prod__NFCI_zscore_60d", "NFCI_vol_20d__ret5x__HD_zscore_60d", "NFCI_ret_5d__div__NFCI_vol_20d", "vix_zscore_10d__zrel__HON_Honeywell_zscore_60d", "NFCI_zscore_60d__minus__vix_ma_20", "heston_ev_h3", "TED_Spread_vol_20d", "NFCI_vol_20d__ret5x__VVIX_vol_20d", "NFCI_ret_5d__div__vix_level", "US30Y_Rate_ret_20d", "EWM_Malaysia_zscore_60d"], "is_new": false}, {"model_id": "v1_h5_GLOBAL_GradientBoosting_N15", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 5, "n_features": 15, "F1_dir": 0.5775, "F1_UP_FORT": 0.3689, "F1_DOWN_FORT": 0.4512, "train_start": "2001-02-07", "sampler": "SMOTEENN", "best_params": "{}", "features": ["NFCI_ret_5d__div__vix_vol_of_vol_10d", "SJM_JM_Smucker_ret_5d", "NFCI_ret_5d__minus__vix_level", "vix_vol_of_vol_10d__prod__NFCI_zscore_60d", "NFCI_vol_20d__ret5x__HD_zscore_60d", "NFCI_ret_5d__div__NFCI_vol_20d", "vix_zscore_10d__zrel__HON_Honeywell_zscore_60d", "NFCI_zscore_60d__minus__vix_ma_20", "heston_ev_h3", "TED_Spread_vol_20d", "NFCI_vol_20d__ret5x__VVIX_vol_20d", "NFCI_ret_5d__div__vix_level", "US30Y_Rate_ret_20d", "EWM_Malaysia_zscore_60d", "PAYX_Paychex_vol_20d"], "is_new": false}, {"model_id": "v1_h5_GLOBAL_RandomForest_N5", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 5, "n_features": 5, "F1_dir": 0.583, "F1_UP_FORT": 0.3328, "F1_DOWN_FORT": 0.3634, "train_start": "2001-02-07", "sampler": "SMOTEENN", "best_params": "{}", "features": ["NFCI_ret_5d__div__vix_vol_of_vol_10d", "SJM_JM_Smucker_ret_5d", "NFCI_ret_5d__minus__vix_level", "vix_vol_of_vol_10d__prod__NFCI_zscore_60d", "NFCI_vol_20d__ret5x__HD_zscore_60d"], "is_new": false}, {"model_id": "v1_h5_GLOBAL_RandomForest_N6", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 5, "n_features": 6, "F1_dir": 0.5917, "F1_UP_FORT": 0.3463, "F1_DOWN_FORT": 0.3606, "train_start": "2001-02-07", "sampler": "SMOTEENN", "best_params": "{}", "features": ["NFCI_ret_5d__div__vix_vol_of_vol_10d", "SJM_JM_Smucker_ret_5d", "NFCI_ret_5d__minus__vix_level", "vix_vol_of_vol_10d__prod__NFCI_zscore_60d", "NFCI_vol_20d__ret5x__HD_zscore_60d", "NFCI_ret_5d__div__NFCI_vol_20d"], "is_new": false}, {"model_id": "v1_h5_GLOBAL_RandomForest_N7", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 5, "n_features": 7, "F1_dir": 0.5993, "F1_UP_FORT": 0.3201, "F1_DOWN_FORT": 0.4081, "train_start": "2001-02-07", "sampler": "SMOTEENN", "best_params": "{}", "features": ["NFCI_ret_5d__div__vix_vol_of_vol_10d", "SJM_JM_Smucker_ret_5d", "NFCI_ret_5d__minus__vix_level", "vix_vol_of_vol_10d__prod__NFCI_zscore_60d", "NFCI_vol_20d__ret5x__HD_zscore_60d", "NFCI_ret_5d__div__NFCI_vol_20d", "vix_zscore_10d__zrel__HON_Honeywell_zscore_60d"], "is_new": false}, {"model_id": "v1_h5_GLOBAL_RandomForest_N8", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 5, "n_features": 8, "F1_dir": 0.6091, "F1_UP_FORT": 0.3636, "F1_DOWN_FORT": 0.4194, "train_start": "2001-02-07", "sampler": "SMOTEENN", "best_params": "{}", "features": ["NFCI_ret_5d__div__vix_vol_of_vol_10d", "SJM_JM_Smucker_ret_5d", "NFCI_ret_5d__minus__vix_level", "vix_vol_of_vol_10d__prod__NFCI_zscore_60d", "NFCI_vol_20d__ret5x__HD_zscore_60d", "NFCI_ret_5d__div__NFCI_vol_20d", "vix_zscore_10d__zrel__HON_Honeywell_zscore_60d", "NFCI_zscore_60d__minus__vix_ma_20"], "is_new": false}, {"model_id": "v1_h5_GLOBAL_RandomForest_N9", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 5, "n_features": 9, "F1_dir": 0.5876, "F1_UP_FORT": 0.2739, "F1_DOWN_FORT": 0.4137, "train_start": "2001-02-07", "sampler": "SMOTEENN", "best_params": "{}", "features": ["NFCI_ret_5d__div__vix_vol_of_vol_10d", "SJM_JM_Smucker_ret_5d", "NFCI_ret_5d__minus__vix_level", "vix_vol_of_vol_10d__prod__NFCI_zscore_60d", "NFCI_vol_20d__ret5x__HD_zscore_60d", "NFCI_ret_5d__div__NFCI_vol_20d", "vix_zscore_10d__zrel__HON_Honeywell_zscore_60d", "NFCI_zscore_60d__minus__vix_ma_20", "heston_ev_h3"], "is_new": false}, {"model_id": "v1_h5_GLOBAL_RandomForest_N10", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 5, "n_features": 10, "F1_dir": 0.5866, "F1_UP_FORT": 0.268, "F1_DOWN_FORT": 0.4372, "train_start": "2001-02-07", "sampler": "SMOTEENN", "best_params": "{}", "features": ["NFCI_ret_5d__div__vix_vol_of_vol_10d", "SJM_JM_Smucker_ret_5d", "NFCI_ret_5d__minus__vix_level", "vix_vol_of_vol_10d__prod__NFCI_zscore_60d", "NFCI_vol_20d__ret5x__HD_zscore_60d", "NFCI_ret_5d__div__NFCI_vol_20d", "vix_zscore_10d__zrel__HON_Honeywell_zscore_60d", "NFCI_zscore_60d__minus__vix_ma_20", "heston_ev_h3", "TED_Spread_vol_20d"], "is_new": false}, {"model_id": "v1_h5_GLOBAL_RandomForest_N11", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 5, "n_features": 11, "F1_dir": 0.5985, "F1_UP_FORT": 0.344, "F1_DOWN_FORT": 0.4331, "train_start": "2001-02-07", "sampler": "SMOTEENN", "best_params": "{}", "features": ["NFCI_ret_5d__div__vix_vol_of_vol_10d", "SJM_JM_Smucker_ret_5d", "NFCI_ret_5d__minus__vix_level", "vix_vol_of_vol_10d__prod__NFCI_zscore_60d", "NFCI_vol_20d__ret5x__HD_zscore_60d", "NFCI_ret_5d__div__NFCI_vol_20d", "vix_zscore_10d__zrel__HON_Honeywell_zscore_60d", "NFCI_zscore_60d__minus__vix_ma_20", "heston_ev_h3", "TED_Spread_vol_20d", "NFCI_vol_20d__ret5x__VVIX_vol_20d"], "is_new": false}, {"model_id": "v1_h5_GLOBAL_RandomForest_N12", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 5, "n_features": 12, "F1_dir": 0.5821, "F1_UP_FORT": 0.2852, "F1_DOWN_FORT": 0.4331, "train_start": "2001-02-07", "sampler": "SMOTEENN", "best_params": "{}", "features": ["NFCI_ret_5d__div__vix_vol_of_vol_10d", "SJM_JM_Smucker_ret_5d", "NFCI_ret_5d__minus__vix_level", "vix_vol_of_vol_10d__prod__NFCI_zscore_60d", "NFCI_vol_20d__ret5x__HD_zscore_60d", "NFCI_ret_5d__div__NFCI_vol_20d", "vix_zscore_10d__zrel__HON_Honeywell_zscore_60d", "NFCI_zscore_60d__minus__vix_ma_20", "heston_ev_h3", "TED_Spread_vol_20d", "NFCI_vol_20d__ret5x__VVIX_vol_20d", "NFCI_ret_5d__div__vix_level"], "is_new": false}, {"model_id": "v1_h5_GLOBAL_RandomForest_N13", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 5, "n_features": 13, "F1_dir": 0.5799, "F1_UP_FORT": 0.2927, "F1_DOWN_FORT": 0.4199, "train_start": "2001-02-07", "sampler": "SMOTEENN", "best_params": "{}", "features": ["NFCI_ret_5d__div__vix_vol_of_vol_10d", "SJM_JM_Smucker_ret_5d", "NFCI_ret_5d__minus__vix_level", "vix_vol_of_vol_10d__prod__NFCI_zscore_60d", "NFCI_vol_20d__ret5x__HD_zscore_60d", "NFCI_ret_5d__div__NFCI_vol_20d", "vix_zscore_10d__zrel__HON_Honeywell_zscore_60d", "NFCI_zscore_60d__minus__vix_ma_20", "heston_ev_h3", "TED_Spread_vol_20d", "NFCI_vol_20d__ret5x__VVIX_vol_20d", "NFCI_ret_5d__div__vix_level", "US30Y_Rate_ret_20d"], "is_new": false}, {"model_id": "v1_h5_GLOBAL_RandomForest_N14", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 5, "n_features": 14, "F1_dir": 0.6024, "F1_UP_FORT": 0.3244, "F1_DOWN_FORT": 0.4449, "train_start": "2001-02-07", "sampler": "SMOTEENN", "best_params": "{}", "features": ["NFCI_ret_5d__div__vix_vol_of_vol_10d", "SJM_JM_Smucker_ret_5d", "NFCI_ret_5d__minus__vix_level", "vix_vol_of_vol_10d__prod__NFCI_zscore_60d", "NFCI_vol_20d__ret5x__HD_zscore_60d", "NFCI_ret_5d__div__NFCI_vol_20d", "vix_zscore_10d__zrel__HON_Honeywell_zscore_60d", "NFCI_zscore_60d__minus__vix_ma_20", "heston_ev_h3", "TED_Spread_vol_20d", "NFCI_vol_20d__ret5x__VVIX_vol_20d", "NFCI_ret_5d__div__vix_level", "US30Y_Rate_ret_20d", "EWM_Malaysia_zscore_60d"], "is_new": false}, {"model_id": "v1_h5_GLOBAL_RandomForest_N15", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 5, "n_features": 15, "F1_dir": 0.599, "F1_UP_FORT": 0.2838, "F1_DOWN_FORT": 0.4442, "train_start": "2001-02-07", "sampler": "SMOTEENN", "best_params": "{}", "features": ["NFCI_ret_5d__div__vix_vol_of_vol_10d", "SJM_JM_Smucker_ret_5d", "NFCI_ret_5d__minus__vix_level", "vix_vol_of_vol_10d__prod__NFCI_zscore_60d", "NFCI_vol_20d__ret5x__HD_zscore_60d", "NFCI_ret_5d__div__NFCI_vol_20d", "vix_zscore_10d__zrel__HON_Honeywell_zscore_60d", "NFCI_zscore_60d__minus__vix_ma_20", "heston_ev_h3", "TED_Spread_vol_20d", "NFCI_vol_20d__ret5x__VVIX_vol_20d", "NFCI_ret_5d__div__vix_level", "US30Y_Rate_ret_20d", "EWM_Malaysia_zscore_60d", "PAYX_Paychex_vol_20d"], "is_new": false}, {"model_id": "v1_h5_GLOBAL_LogisticRegression_N5", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 5, "n_features": 5, "F1_dir": 0.6033, "F1_UP_FORT": 0.3558, "F1_DOWN_FORT": 0.3411, "train_start": "2001-02-07", "sampler": "SMOTEENN", "best_params": "{}", "features": ["NFCI_ret_5d__div__vix_vol_of_vol_10d", "SJM_JM_Smucker_ret_5d", "NFCI_ret_5d__minus__vix_level", "vix_vol_of_vol_10d__prod__NFCI_zscore_60d", "NFCI_vol_20d__ret5x__HD_zscore_60d"], "is_new": false}, {"model_id": "v1_h5_GLOBAL_LogisticRegression_N6", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 5, "n_features": 6, "F1_dir": 0.5989, "F1_UP_FORT": 0.375, "F1_DOWN_FORT": 0.3333, "train_start": "2001-02-07", "sampler": "SMOTEENN", "best_params": "{}", "features": ["NFCI_ret_5d__div__vix_vol_of_vol_10d", "SJM_JM_Smucker_ret_5d", "NFCI_ret_5d__minus__vix_level", "vix_vol_of_vol_10d__prod__NFCI_zscore_60d", "NFCI_vol_20d__ret5x__HD_zscore_60d", "NFCI_ret_5d__div__NFCI_vol_20d"], "is_new": false}, {"model_id": "v1_h5_GLOBAL_LogisticRegression_N7", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 5, "n_features": 7, "F1_dir": 0.5973, "F1_UP_FORT": 0.3213, "F1_DOWN_FORT": 0.4231, "train_start": "2001-02-07", "sampler": "SMOTEENN", "best_params": "{}", "features": ["NFCI_ret_5d__div__vix_vol_of_vol_10d", "SJM_JM_Smucker_ret_5d", "NFCI_ret_5d__minus__vix_level", "vix_vol_of_vol_10d__prod__NFCI_zscore_60d", "NFCI_vol_20d__ret5x__HD_zscore_60d", "NFCI_ret_5d__div__NFCI_vol_20d", "vix_zscore_10d__zrel__HON_Honeywell_zscore_60d"], "is_new": false}, {"model_id": "v1_h5_GLOBAL_LogisticRegression_N8", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 5, "n_features": 8, "F1_dir": 0.5975, "F1_UP_FORT": 0.3065, "F1_DOWN_FORT": 0.4345, "train_start": "2001-02-07", "sampler": "SMOTEENN", "best_params": "{}", "features": ["NFCI_ret_5d__div__vix_vol_of_vol_10d", "SJM_JM_Smucker_ret_5d", "NFCI_ret_5d__minus__vix_level", "vix_vol_of_vol_10d__prod__NFCI_zscore_60d", "NFCI_vol_20d__ret5x__HD_zscore_60d", "NFCI_ret_5d__div__NFCI_vol_20d", "vix_zscore_10d__zrel__HON_Honeywell_zscore_60d", "NFCI_zscore_60d__minus__vix_ma_20"], "is_new": false}, {"model_id": "v1_h5_GLOBAL_LogisticRegression_N9", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 5, "n_features": 9, "F1_dir": 0.5881, "F1_UP_FORT": 0.3233, "F1_DOWN_FORT": 0.4269, "train_start": "2001-02-07", "sampler": "SMOTEENN", "best_params": "{}", "features": ["NFCI_ret_5d__div__vix_vol_of_vol_10d", "SJM_JM_Smucker_ret_5d", "NFCI_ret_5d__minus__vix_level", "vix_vol_of_vol_10d__prod__NFCI_zscore_60d", "NFCI_vol_20d__ret5x__HD_zscore_60d", "NFCI_ret_5d__div__NFCI_vol_20d", "vix_zscore_10d__zrel__HON_Honeywell_zscore_60d", "NFCI_zscore_60d__minus__vix_ma_20", "heston_ev_h3"], "is_new": false}, {"model_id": "v1_h5_GLOBAL_LogisticRegression_N10", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 5, "n_features": 10, "F1_dir": 0.5876, "F1_UP_FORT": 0.31, "F1_DOWN_FORT": 0.4304, "train_start": "2001-02-07", "sampler": "SMOTEENN", "best_params": "{}", "features": ["NFCI_ret_5d__div__vix_vol_of_vol_10d", "SJM_JM_Smucker_ret_5d", "NFCI_ret_5d__minus__vix_level", "vix_vol_of_vol_10d__prod__NFCI_zscore_60d", "NFCI_vol_20d__ret5x__HD_zscore_60d", "NFCI_ret_5d__div__NFCI_vol_20d", "vix_zscore_10d__zrel__HON_Honeywell_zscore_60d", "NFCI_zscore_60d__minus__vix_ma_20", "heston_ev_h3", "TED_Spread_vol_20d"], "is_new": false}, {"model_id": "v1_h5_GLOBAL_LogisticRegression_N11", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 5, "n_features": 11, "F1_dir": 0.5814, "F1_UP_FORT": 0.2945, "F1_DOWN_FORT": 0.4271, "train_start": "2001-02-07", "sampler": "SMOTEENN", "best_params": "{}", "features": ["NFCI_ret_5d__div__vix_vol_of_vol_10d", "SJM_JM_Smucker_ret_5d", "NFCI_ret_5d__minus__vix_level", "vix_vol_of_vol_10d__prod__NFCI_zscore_60d", "NFCI_vol_20d__ret5x__HD_zscore_60d", "NFCI_ret_5d__div__NFCI_vol_20d", "vix_zscore_10d__zrel__HON_Honeywell_zscore_60d", "NFCI_zscore_60d__minus__vix_ma_20", "heston_ev_h3", "TED_Spread_vol_20d", "NFCI_vol_20d__ret5x__VVIX_vol_20d"], "is_new": false}, {"model_id": "v1_h5_GLOBAL_LogisticRegression_N12", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 5, "n_features": 12, "F1_dir": 0.588, "F1_UP_FORT": 0.3005, "F1_DOWN_FORT": 0.426, "train_start": "2001-02-07", "sampler": "SMOTEENN", "best_params": "{}", "features": ["NFCI_ret_5d__div__vix_vol_of_vol_10d", "SJM_JM_Smucker_ret_5d", "NFCI_ret_5d__minus__vix_level", "vix_vol_of_vol_10d__prod__NFCI_zscore_60d", "NFCI_vol_20d__ret5x__HD_zscore_60d", "NFCI_ret_5d__div__NFCI_vol_20d", "vix_zscore_10d__zrel__HON_Honeywell_zscore_60d", "NFCI_zscore_60d__minus__vix_ma_20", "heston_ev_h3", "TED_Spread_vol_20d", "NFCI_vol_20d__ret5x__VVIX_vol_20d", "NFCI_ret_5d__div__vix_level"], "is_new": false}, {"model_id": "v1_h5_GLOBAL_LogisticRegression_N13", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 5, "n_features": 13, "F1_dir": 0.5835, "F1_UP_FORT": 0.323, "F1_DOWN_FORT": 0.397, "train_start": "2001-02-07", "sampler": "SMOTEENN", "best_params": "{}", "features": ["NFCI_ret_5d__div__vix_vol_of_vol_10d", "SJM_JM_Smucker_ret_5d", "NFCI_ret_5d__minus__vix_level", "vix_vol_of_vol_10d__prod__NFCI_zscore_60d", "NFCI_vol_20d__ret5x__HD_zscore_60d", "NFCI_ret_5d__div__NFCI_vol_20d", "vix_zscore_10d__zrel__HON_Honeywell_zscore_60d", "NFCI_zscore_60d__minus__vix_ma_20", "heston_ev_h3", "TED_Spread_vol_20d", "NFCI_vol_20d__ret5x__VVIX_vol_20d", "NFCI_ret_5d__div__vix_level", "US30Y_Rate_ret_20d"], "is_new": false}, {"model_id": "v1_h5_GLOBAL_LogisticRegression_N14", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 5, "n_features": 14, "F1_dir": 0.5751, "F1_UP_FORT": 0.3236, "F1_DOWN_FORT": 0.4053, "train_start": "2001-02-07", "sampler": "SMOTEENN", "best_params": "{}", "features": ["NFCI_ret_5d__div__vix_vol_of_vol_10d", "SJM_JM_Smucker_ret_5d", "NFCI_ret_5d__minus__vix_level", "vix_vol_of_vol_10d__prod__NFCI_zscore_60d", "NFCI_vol_20d__ret5x__HD_zscore_60d", "NFCI_ret_5d__div__NFCI_vol_20d", "vix_zscore_10d__zrel__HON_Honeywell_zscore_60d", "NFCI_zscore_60d__minus__vix_ma_20", "heston_ev_h3", "TED_Spread_vol_20d", "NFCI_vol_20d__ret5x__VVIX_vol_20d", "NFCI_ret_5d__div__vix_level", "US30Y_Rate_ret_20d", "EWM_Malaysia_zscore_60d"], "is_new": false}, {"model_id": "v1_h5_GLOBAL_LogisticRegression_N15", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 5, "n_features": 15, "F1_dir": 0.5763, "F1_UP_FORT": 0.314, "F1_DOWN_FORT": 0.4035, "train_start": "2001-02-07", "sampler": "SMOTEENN", "best_params": "{}", "features": ["NFCI_ret_5d__div__vix_vol_of_vol_10d", "SJM_JM_Smucker_ret_5d", "NFCI_ret_5d__minus__vix_level", "vix_vol_of_vol_10d__prod__NFCI_zscore_60d", "NFCI_vol_20d__ret5x__HD_zscore_60d", "NFCI_ret_5d__div__NFCI_vol_20d", "vix_zscore_10d__zrel__HON_Honeywell_zscore_60d", "NFCI_zscore_60d__minus__vix_ma_20", "heston_ev_h3", "TED_Spread_vol_20d", "NFCI_vol_20d__ret5x__VVIX_vol_20d", "NFCI_ret_5d__div__vix_level", "US30Y_Rate_ret_20d", "EWM_Malaysia_zscore_60d", "PAYX_Paychex_vol_20d"], "is_new": false}, {"model_id": "v1_h5_GLOBAL_RandomForest_Optuna_N8", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 5, "n_features": 8, "F1_dir": 0.5831, "F1_UP_FORT": 0.3901, "F1_DOWN_FORT": 0.4312, "train_start": "2001-02-07", "sampler": "SMOTEENN", "best_params": "{}", "features": ["NFCI_ret_5d__div__vix_vol_of_vol_10d", "SJM_JM_Smucker_ret_5d", "NFCI_ret_5d__minus__vix_level", "vix_vol_of_vol_10d__prod__NFCI_zscore_60d", "NFCI_vol_20d__ret5x__HD_zscore_60d", "NFCI_ret_5d__div__NFCI_vol_20d", "vix_zscore_10d__zrel__HON_Honeywell_zscore_60d", "NFCI_zscore_60d__minus__vix_ma_20"], "is_new": false}, {"model_id": "v1_h5_GLOBAL_RandomForest_OptunaCal_N8", "algo": "RandomForestCal", "regime": "GLOBAL", "horizon": 5, "n_features": 8, "F1_dir": 0.5747, "F1_UP_FORT": 0.3595, "F1_DOWN_FORT": 0.3961, "train_start": "2001-02-07", "sampler": "SMOTEENN", "best_params": "{}", "features": ["NFCI_ret_5d__div__vix_vol_of_vol_10d", "SJM_JM_Smucker_ret_5d", "NFCI_ret_5d__minus__vix_level", "vix_vol_of_vol_10d__prod__NFCI_zscore_60d", "NFCI_vol_20d__ret5x__HD_zscore_60d", "NFCI_ret_5d__div__NFCI_vol_20d", "vix_zscore_10d__zrel__HON_Honeywell_zscore_60d", "NFCI_zscore_60d__minus__vix_ma_20"], "is_new": false}, {"model_id": "v1_h7_CALM_XGBoost_N5", "algo": "XGBoost", "regime": "CALM", "horizon": 7, "n_features": 5, "F1_dir": 0.5926, "F1_UP_FORT": 0.3182, "F1_DOWN_FORT": 0.2718, "train_start": "2000-11-16", "sampler": "ADASYN", "best_params": "{}", "features": ["NFCI_ret_5d__div__NVDA_vol_20d", "NVDA_vol_20d__prod__ADBE_vol_20d", "US3Y_Rate_ret_5d", "XLF_Fin_vol_20d", "Retail_Sales_zscore_60d"], "is_new": false}, {"model_id": "v1_h7_CALM_XGBoost_N6", "algo": "XGBoost", "regime": "CALM", "horizon": 7, "n_features": 6, "F1_dir": 0.5822, "F1_UP_FORT": 0.2653, "F1_DOWN_FORT": 0.3509, "train_start": "2000-11-16", "sampler": "ADASYN", "best_params": "{}", "features": ["NFCI_ret_5d__div__NVDA_vol_20d", "NVDA_vol_20d__prod__ADBE_vol_20d", "US3Y_Rate_ret_5d", "XLF_Fin_vol_20d", "Retail_Sales_zscore_60d", "vix_vs_ma10__zrel__GD_GeneralDynamics_ret_20d"], "is_new": false}, {"model_id": "v1_h7_CALM_XGBoost_N7", "algo": "XGBoost", "regime": "CALM", "horizon": 7, "n_features": 7, "F1_dir": 0.5481, "F1_UP_FORT": 0.3043, "F1_DOWN_FORT": 0.1446, "train_start": "2000-11-16", "sampler": "ADASYN", "best_params": "{}", "features": ["NFCI_ret_5d__div__NVDA_vol_20d", "NVDA_vol_20d__prod__ADBE_vol_20d", "US3Y_Rate_ret_5d", "XLF_Fin_vol_20d", "Retail_Sales_zscore_60d", "vix_vs_ma10__zrel__GD_GeneralDynamics_ret_20d", "MSTR_Bitcoin3_ret_20d"], "is_new": false}, {"model_id": "v1_h7_CALM_XGBoost_N8", "algo": "XGBoost", "regime": "CALM", "horizon": 7, "n_features": 8, "F1_dir": 0.5658, "F1_UP_FORT": 0.2391, "F1_DOWN_FORT": 0.1348, "train_start": "2000-11-16", "sampler": "ADASYN", "best_params": "{}", "features": ["NFCI_ret_5d__div__NVDA_vol_20d", "NVDA_vol_20d__prod__ADBE_vol_20d", "US3Y_Rate_ret_5d", "XLF_Fin_vol_20d", "Retail_Sales_zscore_60d", "vix_vs_ma10__zrel__GD_GeneralDynamics_ret_20d", "MSTR_Bitcoin3_ret_20d", "EWG_Germany_ret_20d"], "is_new": false}, {"model_id": "v1_h7_CALM_XGBoost_N9", "algo": "XGBoost", "regime": "CALM", "horizon": 7, "n_features": 9, "F1_dir": 0.5531, "F1_UP_FORT": 0.2381, "F1_DOWN_FORT": 0.1395, "train_start": "2000-11-16", "sampler": "ADASYN", "best_params": "{}", "features": ["NFCI_ret_5d__div__NVDA_vol_20d", "NVDA_vol_20d__prod__ADBE_vol_20d", "US3Y_Rate_ret_5d", "XLF_Fin_vol_20d", "Retail_Sales_zscore_60d", "vix_vs_ma10__zrel__GD_GeneralDynamics_ret_20d", "MSTR_Bitcoin3_ret_20d", "EWG_Germany_ret_20d", "NFCI_ret_5d__ret5x__QCOM_ret_20d"], "is_new": false}, {"model_id": "v1_h7_CALM_XGBoost_N10", "algo": "XGBoost", "regime": "CALM", "horizon": 7, "n_features": 10, "F1_dir": 0.5619, "F1_UP_FORT": 0.2619, "F1_DOWN_FORT": 0.1176, "train_start": "2000-11-16", "sampler": "ADASYN", "best_params": "{}", "features": ["NFCI_ret_5d__div__NVDA_vol_20d", "NVDA_vol_20d__prod__ADBE_vol_20d", "US3Y_Rate_ret_5d", "XLF_Fin_vol_20d", "Retail_Sales_zscore_60d", "vix_vs_ma10__zrel__GD_GeneralDynamics_ret_20d", "MSTR_Bitcoin3_ret_20d", "EWG_Germany_ret_20d", "NFCI_ret_5d__ret5x__QCOM_ret_20d", "Nikkei_Japan_zscore_60d__div__WFC_WellsFargo_ret_1d"], "is_new": false}, {"model_id": "v1_h7_CALM_XGBoost_N11", "algo": "XGBoost", "regime": "CALM", "horizon": 7, "n_features": 11, "F1_dir": 0.55, "F1_UP_FORT": 0.3023, "F1_DOWN_FORT": 0.1304, "train_start": "2000-11-16", "sampler": "ADASYN", "best_params": "{}", "features": ["NFCI_ret_5d__div__NVDA_vol_20d", "NVDA_vol_20d__prod__ADBE_vol_20d", "US3Y_Rate_ret_5d", "XLF_Fin_vol_20d", "Retail_Sales_zscore_60d", "vix_vs_ma10__zrel__GD_GeneralDynamics_ret_20d", "MSTR_Bitcoin3_ret_20d", "EWG_Germany_ret_20d", "NFCI_ret_5d__ret5x__QCOM_ret_20d", "Nikkei_Japan_zscore_60d__div__WFC_WellsFargo_ret_1d", "HD_ret_5d"], "is_new": false}, {"model_id": "v1_h7_CALM_XGBoost_N12", "algo": "XGBoost", "regime": "CALM", "horizon": 7, "n_features": 12, "F1_dir": 0.5668, "F1_UP_FORT": 0.2927, "F1_DOWN_FORT": 0.1616, "train_start": "2000-11-16", "sampler": "ADASYN", "best_params": "{}", "features": ["NFCI_ret_5d__div__NVDA_vol_20d", "NVDA_vol_20d__prod__ADBE_vol_20d", "US3Y_Rate_ret_5d", "XLF_Fin_vol_20d", "Retail_Sales_zscore_60d", "vix_vs_ma10__zrel__GD_GeneralDynamics_ret_20d", "MSTR_Bitcoin3_ret_20d", "EWG_Germany_ret_20d", "NFCI_ret_5d__ret5x__QCOM_ret_20d", "Nikkei_Japan_zscore_60d__div__WFC_WellsFargo_ret_1d", "HD_ret_5d", "VOD_Vodafone_zscore_60d"], "is_new": false}, {"model_id": "v1_h7_CALM_XGBoost_N13", "algo": "XGBoost", "regime": "CALM", "horizon": 7, "n_features": 13, "F1_dir": 0.5413, "F1_UP_FORT": 0.2892, "F1_DOWN_FORT": 0.1961, "train_start": "2000-11-16", "sampler": "ADASYN", "best_params": "{}", "features": ["NFCI_ret_5d__div__NVDA_vol_20d", "NVDA_vol_20d__prod__ADBE_vol_20d", "US3Y_Rate_ret_5d", "XLF_Fin_vol_20d", "Retail_Sales_zscore_60d", "vix_vs_ma10__zrel__GD_GeneralDynamics_ret_20d", "MSTR_Bitcoin3_ret_20d", "EWG_Germany_ret_20d", "NFCI_ret_5d__ret5x__QCOM_ret_20d", "Nikkei_Japan_zscore_60d__div__WFC_WellsFargo_ret_1d", "HD_ret_5d", "VOD_Vodafone_zscore_60d", "GD_GeneralDynamics_ret_20d__minus__VIX_Price_ret_5d"], "is_new": false}, {"model_id": "v1_h7_CALM_XGBoost_N14", "algo": "XGBoost", "regime": "CALM", "horizon": 7, "n_features": 14, "F1_dir": 0.5595, "F1_UP_FORT": 0.2697, "F1_DOWN_FORT": 0.1961, "train_start": "2000-11-16", "sampler": "ADASYN", "best_params": "{}", "features": ["NFCI_ret_5d__div__NVDA_vol_20d", "NVDA_vol_20d__prod__ADBE_vol_20d", "US3Y_Rate_ret_5d", "XLF_Fin_vol_20d", "Retail_Sales_zscore_60d", "vix_vs_ma10__zrel__GD_GeneralDynamics_ret_20d", "MSTR_Bitcoin3_ret_20d", "EWG_Germany_ret_20d", "NFCI_ret_5d__ret5x__QCOM_ret_20d", "Nikkei_Japan_zscore_60d__div__WFC_WellsFargo_ret_1d", "HD_ret_5d", "VOD_Vodafone_zscore_60d", "GD_GeneralDynamics_ret_20d__minus__VIX_Price_ret_5d", "NFCI_ret_5d__ret5x__Nikkei_Japan_zscore_60d"], "is_new": false}, {"model_id": "v1_h7_CALM_XGBoost_N15", "algo": "XGBoost", "regime": "CALM", "horizon": 7, "n_features": 15, "F1_dir": 0.5647, "F1_UP_FORT": 0.241, "F1_DOWN_FORT": 0.2424, "train_start": "2000-11-16", "sampler": "ADASYN", "best_params": "{}", "features": ["NFCI_ret_5d__div__NVDA_vol_20d", "NVDA_vol_20d__prod__ADBE_vol_20d", "US3Y_Rate_ret_5d", "XLF_Fin_vol_20d", "Retail_Sales_zscore_60d", "vix_vs_ma10__zrel__GD_GeneralDynamics_ret_20d", "MSTR_Bitcoin3_ret_20d", "EWG_Germany_ret_20d", "NFCI_ret_5d__ret5x__QCOM_ret_20d", "Nikkei_Japan_zscore_60d__div__WFC_WellsFargo_ret_1d", "HD_ret_5d", "VOD_Vodafone_zscore_60d", "GD_GeneralDynamics_ret_20d__minus__VIX_Price_ret_5d", "NFCI_ret_5d__ret5x__Nikkei_Japan_zscore_60d", "AXP_Amex_ret_20d"], "is_new": false}, {"model_id": "v1_h7_CALM_XGBoost_N16", "algo": "XGBoost", "regime": "CALM", "horizon": 7, "n_features": 16, "F1_dir": 0.546, "F1_UP_FORT": 0.2381, "F1_DOWN_FORT": 0.1961, "train_start": "2000-11-16", "sampler": "ADASYN", "best_params": "{}", "features": ["NFCI_ret_5d__div__NVDA_vol_20d", "NVDA_vol_20d__prod__ADBE_vol_20d", "US3Y_Rate_ret_5d", "XLF_Fin_vol_20d", "Retail_Sales_zscore_60d", "vix_vs_ma10__zrel__GD_GeneralDynamics_ret_20d", "MSTR_Bitcoin3_ret_20d", "EWG_Germany_ret_20d", "NFCI_ret_5d__ret5x__QCOM_ret_20d", "Nikkei_Japan_zscore_60d__div__WFC_WellsFargo_ret_1d", "HD_ret_5d", "VOD_Vodafone_zscore_60d", "GD_GeneralDynamics_ret_20d__minus__VIX_Price_ret_5d", "NFCI_ret_5d__ret5x__Nikkei_Japan_zscore_60d", "AXP_Amex_ret_20d", "HangSeng_HK_ret_1d"], "is_new": false}, {"model_id": "v1_h7_CALM_XGBoost_N17", "algo": "XGBoost", "regime": "CALM", "horizon": 7, "n_features": 17, "F1_dir": 0.5642, "F1_UP_FORT": 0.2759, "F1_DOWN_FORT": 0.1978, "train_start": "2000-11-16", "sampler": "ADASYN", "best_params": "{}", "features": ["NFCI_ret_5d__div__NVDA_vol_20d", "NVDA_vol_20d__prod__ADBE_vol_20d", "US3Y_Rate_ret_5d", "XLF_Fin_vol_20d", "Retail_Sales_zscore_60d", "vix_vs_ma10__zrel__GD_GeneralDynamics_ret_20d", "MSTR_Bitcoin3_ret_20d", "EWG_Germany_ret_20d", "NFCI_ret_5d__ret5x__QCOM_ret_20d", "Nikkei_Japan_zscore_60d__div__WFC_WellsFargo_ret_1d", "HD_ret_5d", "VOD_Vodafone_zscore_60d", "GD_GeneralDynamics_ret_20d__minus__VIX_Price_ret_5d", "NFCI_ret_5d__ret5x__Nikkei_Japan_zscore_60d", "AXP_Amex_ret_20d", "HangSeng_HK_ret_1d", "vix_vs_ma10__minus__XLV_Health_ret_20d"], "is_new": false}, {"model_id": "v1_h7_CALM_XGBoost_N18", "algo": "XGBoost", "regime": "CALM", "horizon": 7, "n_features": 18, "F1_dir": 0.5433, "F1_UP_FORT": 0.2353, "F1_DOWN_FORT": 0.2178, "train_start": "2000-11-16", "sampler": "ADASYN", "best_params": "{}", "features": ["NFCI_ret_5d__div__NVDA_vol_20d", "NVDA_vol_20d__prod__ADBE_vol_20d", "US3Y_Rate_ret_5d", "XLF_Fin_vol_20d", "Retail_Sales_zscore_60d", "vix_vs_ma10__zrel__GD_GeneralDynamics_ret_20d", "MSTR_Bitcoin3_ret_20d", "EWG_Germany_ret_20d", "NFCI_ret_5d__ret5x__QCOM_ret_20d", "Nikkei_Japan_zscore_60d__div__WFC_WellsFargo_ret_1d", "HD_ret_5d", "VOD_Vodafone_zscore_60d", "GD_GeneralDynamics_ret_20d__minus__VIX_Price_ret_5d", "NFCI_ret_5d__ret5x__Nikkei_Japan_zscore_60d", "AXP_Amex_ret_20d", "HangSeng_HK_ret_1d", "vix_vs_ma10__minus__XLV_Health_ret_20d", "NWL_Newell_ret_20d"], "is_new": false}, {"model_id": "v1_h7_CALM_LightGBM_N5", "algo": "LightGBM", "regime": "CALM", "horizon": 7, "n_features": 5, "F1_dir": 0.5227, "F1_UP_FORT": 0.3226, "F1_DOWN_FORT": 0.3478, "train_start": "2000-11-16", "sampler": "ADASYN", "best_params": "{}", "features": ["NFCI_ret_5d__div__NVDA_vol_20d", "NVDA_vol_20d__prod__ADBE_vol_20d", "US3Y_Rate_ret_5d", "XLF_Fin_vol_20d", "Retail_Sales_zscore_60d"], "is_new": false}, {"model_id": "v1_h7_CALM_LightGBM_N6", "algo": "LightGBM", "regime": "CALM", "horizon": 7, "n_features": 6, "F1_dir": 0.552, "F1_UP_FORT": 0.3, "F1_DOWN_FORT": 0.2772, "train_start": "2000-11-16", "sampler": "ADASYN", "best_params": "{}", "features": ["NFCI_ret_5d__div__NVDA_vol_20d", "NVDA_vol_20d__prod__ADBE_vol_20d", "US3Y_Rate_ret_5d", "XLF_Fin_vol_20d", "Retail_Sales_zscore_60d", "vix_vs_ma10__zrel__GD_GeneralDynamics_ret_20d"], "is_new": false}, {"model_id": "v1_h7_CALM_LightGBM_N8", "algo": "LightGBM", "regime": "CALM", "horizon": 7, "n_features": 8, "F1_dir": 0.5054, "F1_UP_FORT": 0.2892, "F1_DOWN_FORT": 0.0941, "train_start": "2000-11-16", "sampler": "ADASYN", "best_params": "{}", "features": ["NFCI_ret_5d__div__NVDA_vol_20d", "NVDA_vol_20d__prod__ADBE_vol_20d", "US3Y_Rate_ret_5d", "XLF_Fin_vol_20d", "Retail_Sales_zscore_60d", "vix_vs_ma10__zrel__GD_GeneralDynamics_ret_20d", "MSTR_Bitcoin3_ret_20d", "EWG_Germany_ret_20d"], "is_new": false}, {"model_id": "v1_h7_CALM_LightGBM_N9", "algo": "LightGBM", "regime": "CALM", "horizon": 7, "n_features": 9, "F1_dir": 0.5269, "F1_UP_FORT": 0.2381, "F1_DOWN_FORT": 0.0779, "train_start": "2000-11-16", "sampler": "ADASYN", "best_params": "{}", "features": ["NFCI_ret_5d__div__NVDA_vol_20d", "NVDA_vol_20d__prod__ADBE_vol_20d", "US3Y_Rate_ret_5d", "XLF_Fin_vol_20d", "Retail_Sales_zscore_60d", "vix_vs_ma10__zrel__GD_GeneralDynamics_ret_20d", "MSTR_Bitcoin3_ret_20d", "EWG_Germany_ret_20d", "NFCI_ret_5d__ret5x__QCOM_ret_20d"], "is_new": false}, {"model_id": "v1_h7_CALM_LightGBM_N10", "algo": "LightGBM", "regime": "CALM", "horizon": 7, "n_features": 10, "F1_dir": 0.5238, "F1_UP_FORT": 0.2338, "F1_DOWN_FORT": 0.1928, "train_start": "2000-11-16", "sampler": "ADASYN", "best_params": "{}", "features": ["NFCI_ret_5d__div__NVDA_vol_20d", "NVDA_vol_20d__prod__ADBE_vol_20d", "US3Y_Rate_ret_5d", "XLF_Fin_vol_20d", "Retail_Sales_zscore_60d", "vix_vs_ma10__zrel__GD_GeneralDynamics_ret_20d", "MSTR_Bitcoin3_ret_20d", "EWG_Germany_ret_20d", "NFCI_ret_5d__ret5x__QCOM_ret_20d", "Nikkei_Japan_zscore_60d__div__WFC_WellsFargo_ret_1d"], "is_new": false}, {"model_id": "v1_h7_CALM_LightGBM_N11", "algo": "LightGBM", "regime": "CALM", "horizon": 7, "n_features": 11, "F1_dir": 0.5003, "F1_UP_FORT": 0.3333, "F1_DOWN_FORT": 0.1489, "train_start": "2000-11-16", "sampler": "ADASYN", "best_params": "{}", "features": ["NFCI_ret_5d__div__NVDA_vol_20d", "NVDA_vol_20d__prod__ADBE_vol_20d", "US3Y_Rate_ret_5d", "XLF_Fin_vol_20d", "Retail_Sales_zscore_60d", "vix_vs_ma10__zrel__GD_GeneralDynamics_ret_20d", "MSTR_Bitcoin3_ret_20d", "EWG_Germany_ret_20d", "NFCI_ret_5d__ret5x__QCOM_ret_20d", "Nikkei_Japan_zscore_60d__div__WFC_WellsFargo_ret_1d", "HD_ret_5d"], "is_new": false}, {"model_id": "v1_h7_CALM_LightGBM_N12", "algo": "LightGBM", "regime": "CALM", "horizon": 7, "n_features": 12, "F1_dir": 0.5357, "F1_UP_FORT": 0.2466, "F1_DOWN_FORT": 0.1429, "train_start": "2000-11-16", "sampler": "ADASYN", "best_params": "{}", "features": ["NFCI_ret_5d__div__NVDA_vol_20d", "NVDA_vol_20d__prod__ADBE_vol_20d", "US3Y_Rate_ret_5d", "XLF_Fin_vol_20d", "Retail_Sales_zscore_60d", "vix_vs_ma10__zrel__GD_GeneralDynamics_ret_20d", "MSTR_Bitcoin3_ret_20d", "EWG_Germany_ret_20d", "NFCI_ret_5d__ret5x__QCOM_ret_20d", "Nikkei_Japan_zscore_60d__div__WFC_WellsFargo_ret_1d", "HD_ret_5d", "VOD_Vodafone_zscore_60d"], "is_new": false}, {"model_id": "v1_h7_CALM_LightGBM_N13", "algo": "LightGBM", "regime": "CALM", "horizon": 7, "n_features": 13, "F1_dir": 0.5316, "F1_UP_FORT": 0.2785, "F1_DOWN_FORT": 0.1778, "train_start": "2000-11-16", "sampler": "ADASYN", "best_params": "{}", "features": ["NFCI_ret_5d__div__NVDA_vol_20d", "NVDA_vol_20d__prod__ADBE_vol_20d", "US3Y_Rate_ret_5d", "XLF_Fin_vol_20d", "Retail_Sales_zscore_60d", "vix_vs_ma10__zrel__GD_GeneralDynamics_ret_20d", "MSTR_Bitcoin3_ret_20d", "EWG_Germany_ret_20d", "NFCI_ret_5d__ret5x__QCOM_ret_20d", "Nikkei_Japan_zscore_60d__div__WFC_WellsFargo_ret_1d", "HD_ret_5d", "VOD_Vodafone_zscore_60d", "GD_GeneralDynamics_ret_20d__minus__VIX_Price_ret_5d"], "is_new": false}, {"model_id": "v1_h7_CALM_LightGBM_N14", "algo": "LightGBM", "regime": "CALM", "horizon": 7, "n_features": 14, "F1_dir": 0.5553, "F1_UP_FORT": 0.2353, "F1_DOWN_FORT": 0.1609, "train_start": "2000-11-16", "sampler": "ADASYN", "best_params": "{}", "features": ["NFCI_ret_5d__div__NVDA_vol_20d", "NVDA_vol_20d__prod__ADBE_vol_20d", "US3Y_Rate_ret_5d", "XLF_Fin_vol_20d", "Retail_Sales_zscore_60d", "vix_vs_ma10__zrel__GD_GeneralDynamics_ret_20d", "MSTR_Bitcoin3_ret_20d", "EWG_Germany_ret_20d", "NFCI_ret_5d__ret5x__QCOM_ret_20d", "Nikkei_Japan_zscore_60d__div__WFC_WellsFargo_ret_1d", "HD_ret_5d", "VOD_Vodafone_zscore_60d", "GD_GeneralDynamics_ret_20d__minus__VIX_Price_ret_5d", "NFCI_ret_5d__ret5x__Nikkei_Japan_zscore_60d"], "is_new": false}, {"model_id": "v1_h7_CALM_LightGBM_N15", "algo": "LightGBM", "regime": "CALM", "horizon": 7, "n_features": 15, "F1_dir": 0.5234, "F1_UP_FORT": 0.3117, "F1_DOWN_FORT": 0.2118, "train_start": "2000-11-16", "sampler": "ADASYN", "best_params": "{}", "features": ["NFCI_ret_5d__div__NVDA_vol_20d", "NVDA_vol_20d__prod__ADBE_vol_20d", "US3Y_Rate_ret_5d", "XLF_Fin_vol_20d", "Retail_Sales_zscore_60d", "vix_vs_ma10__zrel__GD_GeneralDynamics_ret_20d", "MSTR_Bitcoin3_ret_20d", "EWG_Germany_ret_20d", "NFCI_ret_5d__ret5x__QCOM_ret_20d", "Nikkei_Japan_zscore_60d__div__WFC_WellsFargo_ret_1d", "HD_ret_5d", "VOD_Vodafone_zscore_60d", "GD_GeneralDynamics_ret_20d__minus__VIX_Price_ret_5d", "NFCI_ret_5d__ret5x__Nikkei_Japan_zscore_60d", "AXP_Amex_ret_20d"], "is_new": false}, {"model_id": "v1_h7_CALM_LightGBM_N16", "algo": "LightGBM", "regime": "CALM", "horizon": 7, "n_features": 16, "F1_dir": 0.5012, "F1_UP_FORT": 0.3256, "F1_DOWN_FORT": 0.1818, "train_start": "2000-11-16", "sampler": "ADASYN", "best_params": "{}", "features": ["NFCI_ret_5d__div__NVDA_vol_20d", "NVDA_vol_20d__prod__ADBE_vol_20d", "US3Y_Rate_ret_5d", "XLF_Fin_vol_20d", "Retail_Sales_zscore_60d", "vix_vs_ma10__zrel__GD_GeneralDynamics_ret_20d", "MSTR_Bitcoin3_ret_20d", "EWG_Germany_ret_20d", "NFCI_ret_5d__ret5x__QCOM_ret_20d", "Nikkei_Japan_zscore_60d__div__WFC_WellsFargo_ret_1d", "HD_ret_5d", "VOD_Vodafone_zscore_60d", "GD_GeneralDynamics_ret_20d__minus__VIX_Price_ret_5d", "NFCI_ret_5d__ret5x__Nikkei_Japan_zscore_60d", "AXP_Amex_ret_20d", "HangSeng_HK_ret_1d"], "is_new": false}, {"model_id": "v1_h7_CALM_GradientBoosting_N5", "algo": "GradientBoosting", "regime": "CALM", "horizon": 7, "n_features": 5, "F1_dir": 0.5492, "F1_UP_FORT": 0.2766, "F1_DOWN_FORT": 0.2963, "train_start": "2000-11-16", "sampler": "ADASYN", "best_params": "{}", "features": ["NFCI_ret_5d__div__NVDA_vol_20d", "NVDA_vol_20d__prod__ADBE_vol_20d", "US3Y_Rate_ret_5d", "XLF_Fin_vol_20d", "Retail_Sales_zscore_60d"], "is_new": false}, {"model_id": "v1_h7_CALM_GradientBoosting_N6", "algo": "GradientBoosting", "regime": "CALM", "horizon": 7, "n_features": 6, "F1_dir": 0.5506, "F1_UP_FORT": 0.3061, "F1_DOWN_FORT": 0.25, "train_start": "2000-11-16", "sampler": "ADASYN", "best_params": "{}", "features": ["NFCI_ret_5d__div__NVDA_vol_20d", "NVDA_vol_20d__prod__ADBE_vol_20d", "US3Y_Rate_ret_5d", "XLF_Fin_vol_20d", "Retail_Sales_zscore_60d", "vix_vs_ma10__zrel__GD_GeneralDynamics_ret_20d"], "is_new": false}, {"model_id": "v1_h7_CALM_GradientBoosting_N7", "algo": "GradientBoosting", "regime": "CALM", "horizon": 7, "n_features": 7, "F1_dir": 0.5327, "F1_UP_FORT": 0.3261, "F1_DOWN_FORT": 0.1463, "train_start": "2000-11-16", "sampler": "ADASYN", "best_params": "{}", "features": ["NFCI_ret_5d__div__NVDA_vol_20d", "NVDA_vol_20d__prod__ADBE_vol_20d", "US3Y_Rate_ret_5d", "XLF_Fin_vol_20d", "Retail_Sales_zscore_60d", "vix_vs_ma10__zrel__GD_GeneralDynamics_ret_20d", "MSTR_Bitcoin3_ret_20d"], "is_new": false}, {"model_id": "v1_h7_CALM_GradientBoosting_N8", "algo": "GradientBoosting", "regime": "CALM", "horizon": 7, "n_features": 8, "F1_dir": 0.5465, "F1_UP_FORT": 0.3059, "F1_DOWN_FORT": 0.1111, "train_start": "2000-11-16", "sampler": "ADASYN", "best_params": "{}", "features": ["NFCI_ret_5d__div__NVDA_vol_20d", "NVDA_vol_20d__prod__ADBE_vol_20d", "US3Y_Rate_ret_5d", "XLF_Fin_vol_20d", "Retail_Sales_zscore_60d", "vix_vs_ma10__zrel__GD_GeneralDynamics_ret_20d", "MSTR_Bitcoin3_ret_20d", "EWG_Germany_ret_20d"], "is_new": false}, {"model_id": "v1_h7_CALM_GradientBoosting_N9", "algo": "GradientBoosting", "regime": "CALM", "horizon": 7, "n_features": 9, "F1_dir": 0.5628, "F1_UP_FORT": 0.3333, "F1_DOWN_FORT": 0.1839, "train_start": "2000-11-16", "sampler": "ADASYN", "best_params": "{}", "features": ["NFCI_ret_5d__div__NVDA_vol_20d", "NVDA_vol_20d__prod__ADBE_vol_20d", "US3Y_Rate_ret_5d", "XLF_Fin_vol_20d", "Retail_Sales_zscore_60d", "vix_vs_ma10__zrel__GD_GeneralDynamics_ret_20d", "MSTR_Bitcoin3_ret_20d", "EWG_Germany_ret_20d", "NFCI_ret_5d__ret5x__QCOM_ret_20d"], "is_new": false}, {"model_id": "v1_h7_CALM_GradientBoosting_N10", "algo": "GradientBoosting", "regime": "CALM", "horizon": 7, "n_features": 10, "F1_dir": 0.5523, "F1_UP_FORT": 0.3095, "F1_DOWN_FORT": 0.1163, "train_start": "2000-11-16", "sampler": "ADASYN", "best_params": "{}", "features": ["NFCI_ret_5d__div__NVDA_vol_20d", "NVDA_vol_20d__prod__ADBE_vol_20d", "US3Y_Rate_ret_5d", "XLF_Fin_vol_20d", "Retail_Sales_zscore_60d", "vix_vs_ma10__zrel__GD_GeneralDynamics_ret_20d", "MSTR_Bitcoin3_ret_20d", "EWG_Germany_ret_20d", "NFCI_ret_5d__ret5x__QCOM_ret_20d", "Nikkei_Japan_zscore_60d__div__WFC_WellsFargo_ret_1d"], "is_new": false}, {"model_id": "v1_h7_CALM_GradientBoosting_N11", "algo": "GradientBoosting", "regime": "CALM", "horizon": 7, "n_features": 11, "F1_dir": 0.5433, "F1_UP_FORT": 0.3415, "F1_DOWN_FORT": 0.14, "train_start": "2000-11-16", "sampler": "ADASYN", "best_params": "{}", "features": ["NFCI_ret_5d__div__NVDA_vol_20d", "NVDA_vol_20d__prod__ADBE_vol_20d", "US3Y_Rate_ret_5d", "XLF_Fin_vol_20d", "Retail_Sales_zscore_60d", "vix_vs_ma10__zrel__GD_GeneralDynamics_ret_20d", "MSTR_Bitcoin3_ret_20d", "EWG_Germany_ret_20d", "NFCI_ret_5d__ret5x__QCOM_ret_20d", "Nikkei_Japan_zscore_60d__div__WFC_WellsFargo_ret_1d", "HD_ret_5d"], "is_new": false}, {"model_id": "v1_h7_CALM_GradientBoosting_N12", "algo": "GradientBoosting", "regime": "CALM", "horizon": 7, "n_features": 12, "F1_dir": 0.5309, "F1_UP_FORT": 0.3409, "F1_DOWN_FORT": 0.1458, "train_start": "2000-11-16", "sampler": "ADASYN", "best_params": "{}", "features": ["NFCI_ret_5d__div__NVDA_vol_20d", "NVDA_vol_20d__prod__ADBE_vol_20d", "US3Y_Rate_ret_5d", "XLF_Fin_vol_20d", "Retail_Sales_zscore_60d", "vix_vs_ma10__zrel__GD_GeneralDynamics_ret_20d", "MSTR_Bitcoin3_ret_20d", "EWG_Germany_ret_20d", "NFCI_ret_5d__ret5x__QCOM_ret_20d", "Nikkei_Japan_zscore_60d__div__WFC_WellsFargo_ret_1d", "HD_ret_5d", "VOD_Vodafone_zscore_60d"], "is_new": false}, {"model_id": "v1_h7_CALM_GradientBoosting_N13", "algo": "GradientBoosting", "regime": "CALM", "horizon": 7, "n_features": 13, "F1_dir": 0.5427, "F1_UP_FORT": 0.3678, "F1_DOWN_FORT": 0.1263, "train_start": "2000-11-16", "sampler": "ADASYN", "best_params": "{}", "features": ["NFCI_ret_5d__div__NVDA_vol_20d", "NVDA_vol_20d__prod__ADBE_vol_20d", "US3Y_Rate_ret_5d", "XLF_Fin_vol_20d", "Retail_Sales_zscore_60d", "vix_vs_ma10__zrel__GD_GeneralDynamics_ret_20d", "MSTR_Bitcoin3_ret_20d", "EWG_Germany_ret_20d", "NFCI_ret_5d__ret5x__QCOM_ret_20d", "Nikkei_Japan_zscore_60d__div__WFC_WellsFargo_ret_1d", "HD_ret_5d", "VOD_Vodafone_zscore_60d", "GD_GeneralDynamics_ret_20d__minus__VIX_Price_ret_5d"], "is_new": false}, {"model_id": "v1_h7_CALM_GradientBoosting_N14", "algo": "GradientBoosting", "regime": "CALM", "horizon": 7, "n_features": 14, "F1_dir": 0.579, "F1_UP_FORT": 0.3059, "F1_DOWN_FORT": 0.2157, "train_start": "2000-11-16", "sampler": "ADASYN", "best_params": "{}", "features": ["NFCI_ret_5d__div__NVDA_vol_20d", "NVDA_vol_20d__prod__ADBE_vol_20d", "US3Y_Rate_ret_5d", "XLF_Fin_vol_20d", "Retail_Sales_zscore_60d", "vix_vs_ma10__zrel__GD_GeneralDynamics_ret_20d", "MSTR_Bitcoin3_ret_20d", "EWG_Germany_ret_20d", "NFCI_ret_5d__ret5x__QCOM_ret_20d", "Nikkei_Japan_zscore_60d__div__WFC_WellsFargo_ret_1d", "HD_ret_5d", "VOD_Vodafone_zscore_60d", "GD_GeneralDynamics_ret_20d__minus__VIX_Price_ret_5d", "NFCI_ret_5d__ret5x__Nikkei_Japan_zscore_60d"], "is_new": false}, {"model_id": "v1_h7_CALM_GradientBoosting_N15", "algo": "GradientBoosting", "regime": "CALM", "horizon": 7, "n_features": 15, "F1_dir": 0.5269, "F1_UP_FORT": 0.3256, "F1_DOWN_FORT": 0.1649, "train_start": "2000-11-16", "sampler": "ADASYN", "best_params": "{}", "features": ["NFCI_ret_5d__div__NVDA_vol_20d", "NVDA_vol_20d__prod__ADBE_vol_20d", "US3Y_Rate_ret_5d", "XLF_Fin_vol_20d", "Retail_Sales_zscore_60d", "vix_vs_ma10__zrel__GD_GeneralDynamics_ret_20d", "MSTR_Bitcoin3_ret_20d", "EWG_Germany_ret_20d", "NFCI_ret_5d__ret5x__QCOM_ret_20d", "Nikkei_Japan_zscore_60d__div__WFC_WellsFargo_ret_1d", "HD_ret_5d", "VOD_Vodafone_zscore_60d", "GD_GeneralDynamics_ret_20d__minus__VIX_Price_ret_5d", "NFCI_ret_5d__ret5x__Nikkei_Japan_zscore_60d", "AXP_Amex_ret_20d"], "is_new": false}, {"model_id": "v1_h7_CALM_GradientBoosting_N16", "algo": "GradientBoosting", "regime": "CALM", "horizon": 7, "n_features": 16, "F1_dir": 0.5386, "F1_UP_FORT": 0.3409, "F1_DOWN_FORT": 0.2553, "train_start": "2000-11-16", "sampler": "ADASYN", "best_params": "{}", "features": ["NFCI_ret_5d__div__NVDA_vol_20d", "NVDA_vol_20d__prod__ADBE_vol_20d", "US3Y_Rate_ret_5d", "XLF_Fin_vol_20d", "Retail_Sales_zscore_60d", "vix_vs_ma10__zrel__GD_GeneralDynamics_ret_20d", "MSTR_Bitcoin3_ret_20d", "EWG_Germany_ret_20d", "NFCI_ret_5d__ret5x__QCOM_ret_20d", "Nikkei_Japan_zscore_60d__div__WFC_WellsFargo_ret_1d", "HD_ret_5d", "VOD_Vodafone_zscore_60d", "GD_GeneralDynamics_ret_20d__minus__VIX_Price_ret_5d", "NFCI_ret_5d__ret5x__Nikkei_Japan_zscore_60d", "AXP_Amex_ret_20d", "HangSeng_HK_ret_1d"], "is_new": false}, {"model_id": "v1_h7_CALM_GradientBoosting_N17", "algo": "GradientBoosting", "regime": "CALM", "horizon": 7, "n_features": 17, "F1_dir": 0.5438, "F1_UP_FORT": 0.2889, "F1_DOWN_FORT": 0.1702, "train_start": "2000-11-16", "sampler": "ADASYN", "best_params": "{}", "features": ["NFCI_ret_5d__div__NVDA_vol_20d", "NVDA_vol_20d__prod__ADBE_vol_20d", "US3Y_Rate_ret_5d", "XLF_Fin_vol_20d", "Retail_Sales_zscore_60d", "vix_vs_ma10__zrel__GD_GeneralDynamics_ret_20d", "MSTR_Bitcoin3_ret_20d", "EWG_Germany_ret_20d", "NFCI_ret_5d__ret5x__QCOM_ret_20d", "Nikkei_Japan_zscore_60d__div__WFC_WellsFargo_ret_1d", "HD_ret_5d", "VOD_Vodafone_zscore_60d", "GD_GeneralDynamics_ret_20d__minus__VIX_Price_ret_5d", "NFCI_ret_5d__ret5x__Nikkei_Japan_zscore_60d", "AXP_Amex_ret_20d", "HangSeng_HK_ret_1d", "vix_vs_ma10__minus__XLV_Health_ret_20d"], "is_new": false}, {"model_id": "v1_h7_CALM_GradientBoosting_N18", "algo": "GradientBoosting", "regime": "CALM", "horizon": 7, "n_features": 18, "F1_dir": 0.542, "F1_UP_FORT": 0.25, "F1_DOWN_FORT": 0.1163, "train_start": "2000-11-16", "sampler": "ADASYN", "best_params": "{}", "features": ["NFCI_ret_5d__div__NVDA_vol_20d", "NVDA_vol_20d__prod__ADBE_vol_20d", "US3Y_Rate_ret_5d", "XLF_Fin_vol_20d", "Retail_Sales_zscore_60d", "vix_vs_ma10__zrel__GD_GeneralDynamics_ret_20d", "MSTR_Bitcoin3_ret_20d", "EWG_Germany_ret_20d", "NFCI_ret_5d__ret5x__QCOM_ret_20d", "Nikkei_Japan_zscore_60d__div__WFC_WellsFargo_ret_1d", "HD_ret_5d", "VOD_Vodafone_zscore_60d", "GD_GeneralDynamics_ret_20d__minus__VIX_Price_ret_5d", "NFCI_ret_5d__ret5x__Nikkei_Japan_zscore_60d", "AXP_Amex_ret_20d", "HangSeng_HK_ret_1d", "vix_vs_ma10__minus__XLV_Health_ret_20d", "NWL_Newell_ret_20d"], "is_new": false}, {"model_id": "v1_h7_CALM_RandomForest_N5", "algo": "RandomForest", "regime": "CALM", "horizon": 7, "n_features": 5, "F1_dir": 0.6129, "F1_UP_FORT": 0.2472, "F1_DOWN_FORT": 0.3684, "train_start": "2000-11-16", "sampler": "ADASYN", "best_params": "{}", "features": ["NFCI_ret_5d__div__NVDA_vol_20d", "NVDA_vol_20d__prod__ADBE_vol_20d", "US3Y_Rate_ret_5d", "XLF_Fin_vol_20d", "Retail_Sales_zscore_60d"], "is_new": false}, {"model_id": "v1_h7_CALM_RandomForest_N6", "algo": "RandomForest", "regime": "CALM", "horizon": 7, "n_features": 6, "F1_dir": 0.5885, "F1_UP_FORT": 0.1538, "F1_DOWN_FORT": 0.3252, "train_start": "2000-11-16", "sampler": "ADASYN", "best_params": "{}", "features": ["NFCI_ret_5d__div__NVDA_vol_20d", "NVDA_vol_20d__prod__ADBE_vol_20d", "US3Y_Rate_ret_5d", "XLF_Fin_vol_20d", "Retail_Sales_zscore_60d", "vix_vs_ma10__zrel__GD_GeneralDynamics_ret_20d"], "is_new": false}, {"model_id": "v1_h7_CALM_RandomForest_N7", "algo": "RandomForest", "regime": "CALM", "horizon": 7, "n_features": 7, "F1_dir": 0.5269, "F1_UP_FORT": 0.1622, "F1_DOWN_FORT": 0.1053, "train_start": "2000-11-16", "sampler": "ADASYN", "best_params": "{}", "features": ["NFCI_ret_5d__div__NVDA_vol_20d", "NVDA_vol_20d__prod__ADBE_vol_20d", "US3Y_Rate_ret_5d", "XLF_Fin_vol_20d", "Retail_Sales_zscore_60d", "vix_vs_ma10__zrel__GD_GeneralDynamics_ret_20d", "MSTR_Bitcoin3_ret_20d"], "is_new": false}, {"model_id": "v1_h7_CALM_RandomForest_N8", "algo": "RandomForest", "regime": "CALM", "horizon": 7, "n_features": 8, "F1_dir": 0.5278, "F1_UP_FORT": 0.1842, "F1_DOWN_FORT": 0.1212, "train_start": "2000-11-16", "sampler": "ADASYN", "best_params": "{}", "features": ["NFCI_ret_5d__div__NVDA_vol_20d", "NVDA_vol_20d__prod__ADBE_vol_20d", "US3Y_Rate_ret_5d", "XLF_Fin_vol_20d", "Retail_Sales_zscore_60d", "vix_vs_ma10__zrel__GD_GeneralDynamics_ret_20d", "MSTR_Bitcoin3_ret_20d", "EWG_Germany_ret_20d"], "is_new": false}, {"model_id": "v1_h7_CALM_RandomForest_N9", "algo": "RandomForest", "regime": "CALM", "horizon": 7, "n_features": 9, "F1_dir": 0.5246, "F1_UP_FORT": 0.1842, "F1_DOWN_FORT": 0.1064, "train_start": "2000-11-16", "sampler": "ADASYN", "best_params": "{}", "features": ["NFCI_ret_5d__div__NVDA_vol_20d", "NVDA_vol_20d__prod__ADBE_vol_20d", "US3Y_Rate_ret_5d", "XLF_Fin_vol_20d", "Retail_Sales_zscore_60d", "vix_vs_ma10__zrel__GD_GeneralDynamics_ret_20d", "MSTR_Bitcoin3_ret_20d", "EWG_Germany_ret_20d", "NFCI_ret_5d__ret5x__QCOM_ret_20d"], "is_new": false}, {"model_id": "v1_h7_CALM_RandomForest_N10", "algo": "RandomForest", "regime": "CALM", "horizon": 7, "n_features": 10, "F1_dir": 0.5159, "F1_UP_FORT": 0.1867, "F1_DOWN_FORT": 0.1087, "train_start": "2000-11-16", "sampler": "ADASYN", "best_params": "{}", "features": ["NFCI_ret_5d__div__NVDA_vol_20d", "NVDA_vol_20d__prod__ADBE_vol_20d", "US3Y_Rate_ret_5d", "XLF_Fin_vol_20d", "Retail_Sales_zscore_60d", "vix_vs_ma10__zrel__GD_GeneralDynamics_ret_20d", "MSTR_Bitcoin3_ret_20d", "EWG_Germany_ret_20d", "NFCI_ret_5d__ret5x__QCOM_ret_20d", "Nikkei_Japan_zscore_60d__div__WFC_WellsFargo_ret_1d"], "is_new": false}, {"model_id": "v1_h7_CALM_RandomForest_N12", "algo": "RandomForest", "regime": "CALM", "horizon": 7, "n_features": 12, "F1_dir": 0.5165, "F1_UP_FORT": 0.1667, "F1_DOWN_FORT": 0.1569, "train_start": "2000-11-16", "sampler": "ADASYN", "best_params": "{}", "features": ["NFCI_ret_5d__div__NVDA_vol_20d", "NVDA_vol_20d__prod__ADBE_vol_20d", "US3Y_Rate_ret_5d", "XLF_Fin_vol_20d", "Retail_Sales_zscore_60d", "vix_vs_ma10__zrel__GD_GeneralDynamics_ret_20d", "MSTR_Bitcoin3_ret_20d", "EWG_Germany_ret_20d", "NFCI_ret_5d__ret5x__QCOM_ret_20d", "Nikkei_Japan_zscore_60d__div__WFC_WellsFargo_ret_1d", "HD_ret_5d", "VOD_Vodafone_zscore_60d"], "is_new": false}, {"model_id": "v1_h7_CALM_RandomForest_N13", "algo": "RandomForest", "regime": "CALM", "horizon": 7, "n_features": 13, "F1_dir": 0.5133, "F1_UP_FORT": 0.1667, "F1_DOWN_FORT": 0.1538, "train_start": "2000-11-16", "sampler": "ADASYN", "best_params": "{}", "features": ["NFCI_ret_5d__div__NVDA_vol_20d", "NVDA_vol_20d__prod__ADBE_vol_20d", "US3Y_Rate_ret_5d", "XLF_Fin_vol_20d", "Retail_Sales_zscore_60d", "vix_vs_ma10__zrel__GD_GeneralDynamics_ret_20d", "MSTR_Bitcoin3_ret_20d", "EWG_Germany_ret_20d", "NFCI_ret_5d__ret5x__QCOM_ret_20d", "Nikkei_Japan_zscore_60d__div__WFC_WellsFargo_ret_1d", "HD_ret_5d", "VOD_Vodafone_zscore_60d", "GD_GeneralDynamics_ret_20d__minus__VIX_Price_ret_5d"], "is_new": false}, {"model_id": "v1_h7_CALM_RandomForest_N14", "algo": "RandomForest", "regime": "CALM", "horizon": 7, "n_features": 14, "F1_dir": 0.5326, "F1_UP_FORT": 0.1867, "F1_DOWN_FORT": 0.1782, "train_start": "2000-11-16", "sampler": "ADASYN", "best_params": "{}", "features": ["NFCI_ret_5d__div__NVDA_vol_20d", "NVDA_vol_20d__prod__ADBE_vol_20d", "US3Y_Rate_ret_5d", "XLF_Fin_vol_20d", "Retail_Sales_zscore_60d", "vix_vs_ma10__zrel__GD_GeneralDynamics_ret_20d", "MSTR_Bitcoin3_ret_20d", "EWG_Germany_ret_20d", "NFCI_ret_5d__ret5x__QCOM_ret_20d", "Nikkei_Japan_zscore_60d__div__WFC_WellsFargo_ret_1d", "HD_ret_5d", "VOD_Vodafone_zscore_60d", "GD_GeneralDynamics_ret_20d__minus__VIX_Price_ret_5d", "NFCI_ret_5d__ret5x__Nikkei_Japan_zscore_60d"], "is_new": false}, {"model_id": "v1_h7_CALM_RandomForest_N15", "algo": "RandomForest", "regime": "CALM", "horizon": 7, "n_features": 15, "F1_dir": 0.5061, "F1_UP_FORT": 0.1842, "F1_DOWN_FORT": 0.1538, "train_start": "2000-11-16", "sampler": "ADASYN", "best_params": "{}", "features": ["NFCI_ret_5d__div__NVDA_vol_20d", "NVDA_vol_20d__prod__ADBE_vol_20d", "US3Y_Rate_ret_5d", "XLF_Fin_vol_20d", "Retail_Sales_zscore_60d", "vix_vs_ma10__zrel__GD_GeneralDynamics_ret_20d", "MSTR_Bitcoin3_ret_20d", "EWG_Germany_ret_20d", "NFCI_ret_5d__ret5x__QCOM_ret_20d", "Nikkei_Japan_zscore_60d__div__WFC_WellsFargo_ret_1d", "HD_ret_5d", "VOD_Vodafone_zscore_60d", "GD_GeneralDynamics_ret_20d__minus__VIX_Price_ret_5d", "NFCI_ret_5d__ret5x__Nikkei_Japan_zscore_60d", "AXP_Amex_ret_20d"], "is_new": false}, {"model_id": "v1_h7_CALM_RandomForest_N16", "algo": "RandomForest", "regime": "CALM", "horizon": 7, "n_features": 16, "F1_dir": 0.5088, "F1_UP_FORT": 0.1867, "F1_DOWN_FORT": 0.1856, "train_start": "2000-11-16", "sampler": "ADASYN", "best_params": "{}", "features": ["NFCI_ret_5d__div__NVDA_vol_20d", "NVDA_vol_20d__prod__ADBE_vol_20d", "US3Y_Rate_ret_5d", "XLF_Fin_vol_20d", "Retail_Sales_zscore_60d", "vix_vs_ma10__zrel__GD_GeneralDynamics_ret_20d", "MSTR_Bitcoin3_ret_20d", "EWG_Germany_ret_20d", "NFCI_ret_5d__ret5x__QCOM_ret_20d", "Nikkei_Japan_zscore_60d__div__WFC_WellsFargo_ret_1d", "HD_ret_5d", "VOD_Vodafone_zscore_60d", "GD_GeneralDynamics_ret_20d__minus__VIX_Price_ret_5d", "NFCI_ret_5d__ret5x__Nikkei_Japan_zscore_60d", "AXP_Amex_ret_20d", "HangSeng_HK_ret_1d"], "is_new": false}, {"model_id": "v1_h7_CALM_RandomForest_N18", "algo": "RandomForest", "regime": "CALM", "horizon": 7, "n_features": 18, "F1_dir": 0.5063, "F1_UP_FORT": 0.1538, "F1_DOWN_FORT": 0.1522, "train_start": "2000-11-16", "sampler": "ADASYN", "best_params": "{}", "features": ["NFCI_ret_5d__div__NVDA_vol_20d", "NVDA_vol_20d__prod__ADBE_vol_20d", "US3Y_Rate_ret_5d", "XLF_Fin_vol_20d", "Retail_Sales_zscore_60d", "vix_vs_ma10__zrel__GD_GeneralDynamics_ret_20d", "MSTR_Bitcoin3_ret_20d", "EWG_Germany_ret_20d", "NFCI_ret_5d__ret5x__QCOM_ret_20d", "Nikkei_Japan_zscore_60d__div__WFC_WellsFargo_ret_1d", "HD_ret_5d", "VOD_Vodafone_zscore_60d", "GD_GeneralDynamics_ret_20d__minus__VIX_Price_ret_5d", "NFCI_ret_5d__ret5x__Nikkei_Japan_zscore_60d", "AXP_Amex_ret_20d", "HangSeng_HK_ret_1d", "vix_vs_ma10__minus__XLV_Health_ret_20d", "NWL_Newell_ret_20d"], "is_new": false}, {"model_id": "v1_h7_CALM_RandomForest_Optuna_N5", "algo": "RandomForest", "regime": "CALM", "horizon": 7, "n_features": 5, "F1_dir": 0.5971, "F1_UP_FORT": 0.1728, "F1_DOWN_FORT": 0.3486, "train_start": "2000-11-16", "sampler": "ADASYN", "best_params": "{}", "features": ["NFCI_ret_5d__div__NVDA_vol_20d", "NVDA_vol_20d__prod__ADBE_vol_20d", "US3Y_Rate_ret_5d", "XLF_Fin_vol_20d", "Retail_Sales_zscore_60d"], "is_new": false}, {"model_id": "v1_h7_CALM_RandomForest_OptunaCal_N5", "algo": "RandomForestCal", "regime": "CALM", "horizon": 7, "n_features": 5, "F1_dir": 0.593, "F1_UP_FORT": 0.0606, "F1_DOWN_FORT": 0.3008, "train_start": "2000-11-16", "sampler": "ADASYN", "best_params": "{}", "features": ["NFCI_ret_5d__div__NVDA_vol_20d", "NVDA_vol_20d__prod__ADBE_vol_20d", "US3Y_Rate_ret_5d", "XLF_Fin_vol_20d", "Retail_Sales_zscore_60d"], "is_new": false}, {"model_id": "v1_h7_NORMAL_XGBoost_N5", "algo": "XGBoost", "regime": "NORMAL", "horizon": 7, "n_features": 5, "F1_dir": 0.5157, "F1_UP_FORT": 0.3452, "F1_DOWN_FORT": 0.232, "train_start": "2000-11-10", "sampler": "SMOTETomek", "best_params": "{}", "features": ["NFCI_ret_5d__prod__CLX_Clorox_vol_20d", "NFCI_ret_5d__minus__NFCI_vol_20d", "Nikkei_Japan_vol_20d", "MKC_McCormick_ret_5d__minus__HangSeng_HK_ret_5d", "NFCI_ret_5d__minus__M_Macys_vol_20d"], "is_new": false}, {"model_id": "v1_h7_NORMAL_XGBoost_N6", "algo": "XGBoost", "regime": "NORMAL", "horizon": 7, "n_features": 6, "F1_dir": 0.534, "F1_UP_FORT": 0.3005, "F1_DOWN_FORT": 0.2946, "train_start": "2000-11-10", "sampler": "SMOTETomek", "best_params": "{}", "features": ["NFCI_ret_5d__prod__CLX_Clorox_vol_20d", "NFCI_ret_5d__minus__NFCI_vol_20d", "Nikkei_Japan_vol_20d", "MKC_McCormick_ret_5d__minus__HangSeng_HK_ret_5d", "NFCI_ret_5d__minus__M_Macys_vol_20d", "LOW_Lowes_ret_20d"], "is_new": false}, {"model_id": "v1_h7_NORMAL_XGBoost_N7", "algo": "XGBoost", "regime": "NORMAL", "horizon": 7, "n_features": 7, "F1_dir": 0.5246, "F1_UP_FORT": 0.3519, "F1_DOWN_FORT": 0.3081, "train_start": "2000-11-10", "sampler": "SMOTETomek", "best_params": "{}", "features": ["NFCI_ret_5d__prod__CLX_Clorox_vol_20d", "NFCI_ret_5d__minus__NFCI_vol_20d", "Nikkei_Japan_vol_20d", "MKC_McCormick_ret_5d__minus__HangSeng_HK_ret_5d", "NFCI_ret_5d__minus__M_Macys_vol_20d", "LOW_Lowes_ret_20d", "CPB_CampbellSoup_ret_20d"], "is_new": false}, {"model_id": "v1_h7_NORMAL_XGBoost_N8", "algo": "XGBoost", "regime": "NORMAL", "horizon": 7, "n_features": 8, "F1_dir": 0.549, "F1_UP_FORT": 0.3333, "F1_DOWN_FORT": 0.3553, "train_start": "2000-11-10", "sampler": "SMOTETomek", "best_params": "{}", "features": ["NFCI_ret_5d__prod__CLX_Clorox_vol_20d", "NFCI_ret_5d__minus__NFCI_vol_20d", "Nikkei_Japan_vol_20d", "MKC_McCormick_ret_5d__minus__HangSeng_HK_ret_5d", "NFCI_ret_5d__minus__M_Macys_vol_20d", "LOW_Lowes_ret_20d", "CPB_CampbellSoup_ret_20d", "NFCI_ret_5d__prod__vix_zscore_10d"], "is_new": false}, {"model_id": "v1_h7_NORMAL_XGBoost_N9", "algo": "XGBoost", "regime": "NORMAL", "horizon": 7, "n_features": 9, "F1_dir": 0.513, "F1_UP_FORT": 0.3412, "F1_DOWN_FORT": 0.2976, "train_start": "2000-11-10", "sampler": "SMOTETomek", "best_params": "{}", "features": ["NFCI_ret_5d__prod__CLX_Clorox_vol_20d", "NFCI_ret_5d__minus__NFCI_vol_20d", "Nikkei_Japan_vol_20d", "MKC_McCormick_ret_5d__minus__HangSeng_HK_ret_5d", "NFCI_ret_5d__minus__M_Macys_vol_20d", "LOW_Lowes_ret_20d", "CPB_CampbellSoup_ret_20d", "NFCI_ret_5d__prod__vix_zscore_10d", "heston_var_ev_h1__minus__M_Macys_vol_20d"], "is_new": false}, {"model_id": "v1_h7_NORMAL_XGBoost_N11", "algo": "XGBoost", "regime": "NORMAL", "horizon": 7, "n_features": 11, "F1_dir": 0.522, "F1_UP_FORT": 0.3437, "F1_DOWN_FORT": 0.3044, "train_start": "2000-11-10", "sampler": "SMOTETomek", "best_params": "{}", "features": ["NFCI_ret_5d__prod__CLX_Clorox_vol_20d", "NFCI_ret_5d__minus__NFCI_vol_20d", "Nikkei_Japan_vol_20d", "MKC_McCormick_ret_5d__minus__HangSeng_HK_ret_5d", "NFCI_ret_5d__minus__M_Macys_vol_20d", "LOW_Lowes_ret_20d", "CPB_CampbellSoup_ret_20d", "NFCI_ret_5d__prod__vix_zscore_10d", "heston_var_ev_h1__minus__M_Macys_vol_20d", "PAYX_Paychex_zscore_60d", "NFCI_ret_5d__minus__CLX_Clorox_vol_20d"], "is_new": false}, {"model_id": "v1_h7_NORMAL_XGBoost_N12", "algo": "XGBoost", "regime": "NORMAL", "horizon": 7, "n_features": 12, "F1_dir": 0.5037, "F1_UP_FORT": 0.349, "F1_DOWN_FORT": 0.3215, "train_start": "2000-11-10", "sampler": "SMOTETomek", "best_params": "{}", "features": ["NFCI_ret_5d__prod__CLX_Clorox_vol_20d", "NFCI_ret_5d__minus__NFCI_vol_20d", "Nikkei_Japan_vol_20d", "MKC_McCormick_ret_5d__minus__HangSeng_HK_ret_5d", "NFCI_ret_5d__minus__M_Macys_vol_20d", "LOW_Lowes_ret_20d", "CPB_CampbellSoup_ret_20d", "NFCI_ret_5d__prod__vix_zscore_10d", "heston_var_ev_h1__minus__M_Macys_vol_20d", "PAYX_Paychex_zscore_60d", "NFCI_ret_5d__minus__CLX_Clorox_vol_20d", "EWG_Germany_vol_20d"], "is_new": false}, {"model_id": "v1_h7_NORMAL_XGBoost_N16", "algo": "XGBoost", "regime": "NORMAL", "horizon": 7, "n_features": 16, "F1_dir": 0.511, "F1_UP_FORT": 0.3294, "F1_DOWN_FORT": 0.3242, "train_start": "2000-11-10", "sampler": "SMOTETomek", "best_params": "{}", "features": ["NFCI_ret_5d__prod__CLX_Clorox_vol_20d", "NFCI_ret_5d__minus__NFCI_vol_20d", "Nikkei_Japan_vol_20d", "MKC_McCormick_ret_5d__minus__HangSeng_HK_ret_5d", "NFCI_ret_5d__minus__M_Macys_vol_20d", "LOW_Lowes_ret_20d", "CPB_CampbellSoup_ret_20d", "NFCI_ret_5d__prod__vix_zscore_10d", "heston_var_ev_h1__minus__M_Macys_vol_20d", "PAYX_Paychex_zscore_60d", "NFCI_ret_5d__minus__CLX_Clorox_vol_20d", "EWG_Germany_vol_20d", "MKC_McCormick_ret_5d__zrel__COST_ret_5d", "MKC_McCormick_ret_5d__minus__COST_ret_5d", "heston_var_ev_h1__minus__CLX_Clorox_vol_20d", "IBEX_Spain_ret_20d"], "is_new": false}, {"model_id": "v1_h7_NORMAL_LightGBM_N5", "algo": "LightGBM", "regime": "NORMAL", "horizon": 7, "n_features": 5, "F1_dir": 0.5389, "F1_UP_FORT": 0.3155, "F1_DOWN_FORT": 0.2282, "train_start": "2000-11-10", "sampler": "SMOTETomek", "best_params": "{}", "features": ["NFCI_ret_5d__prod__CLX_Clorox_vol_20d", "NFCI_ret_5d__minus__NFCI_vol_20d", "Nikkei_Japan_vol_20d", "MKC_McCormick_ret_5d__minus__HangSeng_HK_ret_5d", "NFCI_ret_5d__minus__M_Macys_vol_20d"], "is_new": false}, {"model_id": "v1_h7_NORMAL_LightGBM_N6", "algo": "LightGBM", "regime": "NORMAL", "horizon": 7, "n_features": 6, "F1_dir": 0.5352, "F1_UP_FORT": 0.3069, "F1_DOWN_FORT": 0.2889, "train_start": "2000-11-10", "sampler": "SMOTETomek", "best_params": "{}", "features": ["NFCI_ret_5d__prod__CLX_Clorox_vol_20d", "NFCI_ret_5d__minus__NFCI_vol_20d", "Nikkei_Japan_vol_20d", "MKC_McCormick_ret_5d__minus__HangSeng_HK_ret_5d", "NFCI_ret_5d__minus__M_Macys_vol_20d", "LOW_Lowes_ret_20d"], "is_new": false}, {"model_id": "v1_h7_NORMAL_LightGBM_N7", "algo": "LightGBM", "regime": "NORMAL", "horizon": 7, "n_features": 7, "F1_dir": 0.5208, "F1_UP_FORT": 0.3415, "F1_DOWN_FORT": 0.271, "train_start": "2000-11-10", "sampler": "SMOTETomek", "best_params": "{}", "features": ["NFCI_ret_5d__prod__CLX_Clorox_vol_20d", "NFCI_ret_5d__minus__NFCI_vol_20d", "Nikkei_Japan_vol_20d", "MKC_McCormick_ret_5d__minus__HangSeng_HK_ret_5d", "NFCI_ret_5d__minus__M_Macys_vol_20d", "LOW_Lowes_ret_20d", "CPB_CampbellSoup_ret_20d"], "is_new": false}, {"model_id": "v1_h7_NORMAL_LightGBM_N8", "algo": "LightGBM", "regime": "NORMAL", "horizon": 7, "n_features": 8, "F1_dir": 0.5284, "F1_UP_FORT": 0.3135, "F1_DOWN_FORT": 0.3496, "train_start": "2000-11-10", "sampler": "SMOTETomek", "best_params": "{}", "features": ["NFCI_ret_5d__prod__CLX_Clorox_vol_20d", "NFCI_ret_5d__minus__NFCI_vol_20d", "Nikkei_Japan_vol_20d", "MKC_McCormick_ret_5d__minus__HangSeng_HK_ret_5d", "NFCI_ret_5d__minus__M_Macys_vol_20d", "LOW_Lowes_ret_20d", "CPB_CampbellSoup_ret_20d", "NFCI_ret_5d__prod__vix_zscore_10d"], "is_new": false}, {"model_id": "v1_h7_NORMAL_LightGBM_N9", "algo": "LightGBM", "regime": "NORMAL", "horizon": 7, "n_features": 9, "F1_dir": 0.5097, "F1_UP_FORT": 0.2519, "F1_DOWN_FORT": 0.3178, "train_start": "2000-11-10", "sampler": "SMOTETomek", "best_params": "{}", "features": ["NFCI_ret_5d__prod__CLX_Clorox_vol_20d", "NFCI_ret_5d__minus__NFCI_vol_20d", "Nikkei_Japan_vol_20d", "MKC_McCormick_ret_5d__minus__HangSeng_HK_ret_5d", "NFCI_ret_5d__minus__M_Macys_vol_20d", "LOW_Lowes_ret_20d", "CPB_CampbellSoup_ret_20d", "NFCI_ret_5d__prod__vix_zscore_10d", "heston_var_ev_h1__minus__M_Macys_vol_20d"], "is_new": false}, {"model_id": "v1_h7_NORMAL_LightGBM_N10", "algo": "LightGBM", "regime": "NORMAL", "horizon": 7, "n_features": 10, "F1_dir": 0.5067, "F1_UP_FORT": 0.264, "F1_DOWN_FORT": 0.3293, "train_start": "2000-11-10", "sampler": "SMOTETomek", "best_params": "{}", "features": ["NFCI_ret_5d__prod__CLX_Clorox_vol_20d", "NFCI_ret_5d__minus__NFCI_vol_20d", "Nikkei_Japan_vol_20d", "MKC_McCormick_ret_5d__minus__HangSeng_HK_ret_5d", "NFCI_ret_5d__minus__M_Macys_vol_20d", "LOW_Lowes_ret_20d", "CPB_CampbellSoup_ret_20d", "NFCI_ret_5d__prod__vix_zscore_10d", "heston_var_ev_h1__minus__M_Macys_vol_20d", "PAYX_Paychex_zscore_60d"], "is_new": false}, {"model_id": "v1_h7_NORMAL_LightGBM_N11", "algo": "LightGBM", "regime": "NORMAL", "horizon": 7, "n_features": 11, "F1_dir": 0.5063, "F1_UP_FORT": 0.2878, "F1_DOWN_FORT": 0.3112, "train_start": "2000-11-10", "sampler": "SMOTETomek", "best_params": "{}", "features": ["NFCI_ret_5d__prod__CLX_Clorox_vol_20d", "NFCI_ret_5d__minus__NFCI_vol_20d", "Nikkei_Japan_vol_20d", "MKC_McCormick_ret_5d__minus__HangSeng_HK_ret_5d", "NFCI_ret_5d__minus__M_Macys_vol_20d", "LOW_Lowes_ret_20d", "CPB_CampbellSoup_ret_20d", "NFCI_ret_5d__prod__vix_zscore_10d", "heston_var_ev_h1__minus__M_Macys_vol_20d", "PAYX_Paychex_zscore_60d", "NFCI_ret_5d__minus__CLX_Clorox_vol_20d"], "is_new": false}, {"model_id": "v1_h7_NORMAL_LightGBM_N12", "algo": "LightGBM", "regime": "NORMAL", "horizon": 7, "n_features": 12, "F1_dir": 0.5113, "F1_UP_FORT": 0.337, "F1_DOWN_FORT": 0.3264, "train_start": "2000-11-10", "sampler": "SMOTETomek", "best_params": "{}", "features": ["NFCI_ret_5d__prod__CLX_Clorox_vol_20d", "NFCI_ret_5d__minus__NFCI_vol_20d", "Nikkei_Japan_vol_20d", "MKC_McCormick_ret_5d__minus__HangSeng_HK_ret_5d", "NFCI_ret_5d__minus__M_Macys_vol_20d", "LOW_Lowes_ret_20d", "CPB_CampbellSoup_ret_20d", "NFCI_ret_5d__prod__vix_zscore_10d", "heston_var_ev_h1__minus__M_Macys_vol_20d", "PAYX_Paychex_zscore_60d", "NFCI_ret_5d__minus__CLX_Clorox_vol_20d", "EWG_Germany_vol_20d"], "is_new": false}, {"model_id": "v1_h7_NORMAL_LightGBM_N13", "algo": "LightGBM", "regime": "NORMAL", "horizon": 7, "n_features": 13, "F1_dir": 0.5155, "F1_UP_FORT": 0.3303, "F1_DOWN_FORT": 0.31, "train_start": "2000-11-10", "sampler": "SMOTETomek", "best_params": "{}", "features": ["NFCI_ret_5d__prod__CLX_Clorox_vol_20d", "NFCI_ret_5d__minus__NFCI_vol_20d", "Nikkei_Japan_vol_20d", "MKC_McCormick_ret_5d__minus__HangSeng_HK_ret_5d", "NFCI_ret_5d__minus__M_Macys_vol_20d", "LOW_Lowes_ret_20d", "CPB_CampbellSoup_ret_20d", "NFCI_ret_5d__prod__vix_zscore_10d", "heston_var_ev_h1__minus__M_Macys_vol_20d", "PAYX_Paychex_zscore_60d", "NFCI_ret_5d__minus__CLX_Clorox_vol_20d", "EWG_Germany_vol_20d", "MKC_McCormick_ret_5d__zrel__COST_ret_5d"], "is_new": false}, {"model_id": "v1_h7_NORMAL_LightGBM_N14", "algo": "LightGBM", "regime": "NORMAL", "horizon": 7, "n_features": 14, "F1_dir": 0.505, "F1_UP_FORT": 0.331, "F1_DOWN_FORT": 0.3128, "train_start": "2000-11-10", "sampler": "SMOTETomek", "best_params": "{}", "features": ["NFCI_ret_5d__prod__CLX_Clorox_vol_20d", "NFCI_ret_5d__minus__NFCI_vol_20d", "Nikkei_Japan_vol_20d", "MKC_McCormick_ret_5d__minus__HangSeng_HK_ret_5d", "NFCI_ret_5d__minus__M_Macys_vol_20d", "LOW_Lowes_ret_20d", "CPB_CampbellSoup_ret_20d", "NFCI_ret_5d__prod__vix_zscore_10d", "heston_var_ev_h1__minus__M_Macys_vol_20d", "PAYX_Paychex_zscore_60d", "NFCI_ret_5d__minus__CLX_Clorox_vol_20d", "EWG_Germany_vol_20d", "MKC_McCormick_ret_5d__zrel__COST_ret_5d", "MKC_McCormick_ret_5d__minus__COST_ret_5d"], "is_new": false}, {"model_id": "v1_h7_NORMAL_LightGBM_N15", "algo": "LightGBM", "regime": "NORMAL", "horizon": 7, "n_features": 15, "F1_dir": 0.5239, "F1_UP_FORT": 0.3432, "F1_DOWN_FORT": 0.3186, "train_start": "2000-11-10", "sampler": "SMOTETomek", "best_params": "{}", "features": ["NFCI_ret_5d__prod__CLX_Clorox_vol_20d", "NFCI_ret_5d__minus__NFCI_vol_20d", "Nikkei_Japan_vol_20d", "MKC_McCormick_ret_5d__minus__HangSeng_HK_ret_5d", "NFCI_ret_5d__minus__M_Macys_vol_20d", "LOW_Lowes_ret_20d", "CPB_CampbellSoup_ret_20d", "NFCI_ret_5d__prod__vix_zscore_10d", "heston_var_ev_h1__minus__M_Macys_vol_20d", "PAYX_Paychex_zscore_60d", "NFCI_ret_5d__minus__CLX_Clorox_vol_20d", "EWG_Germany_vol_20d", "MKC_McCormick_ret_5d__zrel__COST_ret_5d", "MKC_McCormick_ret_5d__minus__COST_ret_5d", "heston_var_ev_h1__minus__CLX_Clorox_vol_20d"], "is_new": false}, {"model_id": "v1_h7_NORMAL_LightGBM_N16", "algo": "LightGBM", "regime": "NORMAL", "horizon": 7, "n_features": 16, "F1_dir": 0.5368, "F1_UP_FORT": 0.3704, "F1_DOWN_FORT": 0.358, "train_start": "2000-11-10", "sampler": "SMOTETomek", "best_params": "{}", "features": ["NFCI_ret_5d__prod__CLX_Clorox_vol_20d", "NFCI_ret_5d__minus__NFCI_vol_20d", "Nikkei_Japan_vol_20d", "MKC_McCormick_ret_5d__minus__HangSeng_HK_ret_5d", "NFCI_ret_5d__minus__M_Macys_vol_20d", "LOW_Lowes_ret_20d", "CPB_CampbellSoup_ret_20d", "NFCI_ret_5d__prod__vix_zscore_10d", "heston_var_ev_h1__minus__M_Macys_vol_20d", "PAYX_Paychex_zscore_60d", "NFCI_ret_5d__minus__CLX_Clorox_vol_20d", "EWG_Germany_vol_20d", "MKC_McCormick_ret_5d__zrel__COST_ret_5d", "MKC_McCormick_ret_5d__minus__COST_ret_5d", "heston_var_ev_h1__minus__CLX_Clorox_vol_20d", "IBEX_Spain_ret_20d"], "is_new": false}, {"model_id": "v1_h7_NORMAL_GradientBoosting_N5", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 7, "n_features": 5, "F1_dir": 0.5195, "F1_UP_FORT": 0.3202, "F1_DOWN_FORT": 0.2727, "train_start": "2000-11-10", "sampler": "SMOTETomek", "best_params": "{}", "features": ["NFCI_ret_5d__prod__CLX_Clorox_vol_20d", "NFCI_ret_5d__minus__NFCI_vol_20d", "Nikkei_Japan_vol_20d", "MKC_McCormick_ret_5d__minus__HangSeng_HK_ret_5d", "NFCI_ret_5d__minus__M_Macys_vol_20d"], "is_new": false}, {"model_id": "v1_h7_NORMAL_GradientBoosting_N6", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 7, "n_features": 6, "F1_dir": 0.5388, "F1_UP_FORT": 0.379, "F1_DOWN_FORT": 0.2568, "train_start": "2000-11-10", "sampler": "SMOTETomek", "best_params": "{}", "features": ["NFCI_ret_5d__prod__CLX_Clorox_vol_20d", "NFCI_ret_5d__minus__NFCI_vol_20d", "Nikkei_Japan_vol_20d", "MKC_McCormick_ret_5d__minus__HangSeng_HK_ret_5d", "NFCI_ret_5d__minus__M_Macys_vol_20d", "LOW_Lowes_ret_20d"], "is_new": false}, {"model_id": "v1_h7_NORMAL_GradientBoosting_N7", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 7, "n_features": 7, "F1_dir": 0.5046, "F1_UP_FORT": 0.3263, "F1_DOWN_FORT": 0.252, "train_start": "2000-11-10", "sampler": "SMOTETomek", "best_params": "{}", "features": ["NFCI_ret_5d__prod__CLX_Clorox_vol_20d", "NFCI_ret_5d__minus__NFCI_vol_20d", "Nikkei_Japan_vol_20d", "MKC_McCormick_ret_5d__minus__HangSeng_HK_ret_5d", "NFCI_ret_5d__minus__M_Macys_vol_20d", "LOW_Lowes_ret_20d", "CPB_CampbellSoup_ret_20d"], "is_new": false}, {"model_id": "v1_h7_NORMAL_GradientBoosting_N8", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 7, "n_features": 8, "F1_dir": 0.5358, "F1_UP_FORT": 0.319, "F1_DOWN_FORT": 0.3279, "train_start": "2000-11-10", "sampler": "SMOTETomek", "best_params": "{}", "features": ["NFCI_ret_5d__prod__CLX_Clorox_vol_20d", "NFCI_ret_5d__minus__NFCI_vol_20d", "Nikkei_Japan_vol_20d", "MKC_McCormick_ret_5d__minus__HangSeng_HK_ret_5d", "NFCI_ret_5d__minus__M_Macys_vol_20d", "LOW_Lowes_ret_20d", "CPB_CampbellSoup_ret_20d", "NFCI_ret_5d__prod__vix_zscore_10d"], "is_new": false}, {"model_id": "v1_h7_NORMAL_GradientBoosting_N9", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 7, "n_features": 9, "F1_dir": 0.5259, "F1_UP_FORT": 0.3148, "F1_DOWN_FORT": 0.3232, "train_start": "2000-11-10", "sampler": "SMOTETomek", "best_params": "{}", "features": ["NFCI_ret_5d__prod__CLX_Clorox_vol_20d", "NFCI_ret_5d__minus__NFCI_vol_20d", "Nikkei_Japan_vol_20d", "MKC_McCormick_ret_5d__minus__HangSeng_HK_ret_5d", "NFCI_ret_5d__minus__M_Macys_vol_20d", "LOW_Lowes_ret_20d", "CPB_CampbellSoup_ret_20d", "NFCI_ret_5d__prod__vix_zscore_10d", "heston_var_ev_h1__minus__M_Macys_vol_20d"], "is_new": false}, {"model_id": "v1_h7_NORMAL_GradientBoosting_N11", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 7, "n_features": 11, "F1_dir": 0.5061, "F1_UP_FORT": 0.272, "F1_DOWN_FORT": 0.3256, "train_start": "2000-11-10", "sampler": "SMOTETomek", "best_params": "{}", "features": ["NFCI_ret_5d__prod__CLX_Clorox_vol_20d", "NFCI_ret_5d__minus__NFCI_vol_20d", "Nikkei_Japan_vol_20d", "MKC_McCormick_ret_5d__minus__HangSeng_HK_ret_5d", "NFCI_ret_5d__minus__M_Macys_vol_20d", "LOW_Lowes_ret_20d", "CPB_CampbellSoup_ret_20d", "NFCI_ret_5d__prod__vix_zscore_10d", "heston_var_ev_h1__minus__M_Macys_vol_20d", "PAYX_Paychex_zscore_60d", "NFCI_ret_5d__minus__CLX_Clorox_vol_20d"], "is_new": false}, {"model_id": "v1_h7_NORMAL_GradientBoosting_N14", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 7, "n_features": 14, "F1_dir": 0.5009, "F1_UP_FORT": 0.3387, "F1_DOWN_FORT": 0.3246, "train_start": "2000-11-10", "sampler": "SMOTETomek", "best_params": "{}", "features": ["NFCI_ret_5d__prod__CLX_Clorox_vol_20d", "NFCI_ret_5d__minus__NFCI_vol_20d", "Nikkei_Japan_vol_20d", "MKC_McCormick_ret_5d__minus__HangSeng_HK_ret_5d", "NFCI_ret_5d__minus__M_Macys_vol_20d", "LOW_Lowes_ret_20d", "CPB_CampbellSoup_ret_20d", "NFCI_ret_5d__prod__vix_zscore_10d", "heston_var_ev_h1__minus__M_Macys_vol_20d", "PAYX_Paychex_zscore_60d", "NFCI_ret_5d__minus__CLX_Clorox_vol_20d", "EWG_Germany_vol_20d", "MKC_McCormick_ret_5d__zrel__COST_ret_5d", "MKC_McCormick_ret_5d__minus__COST_ret_5d"], "is_new": false}, {"model_id": "v1_h7_NORMAL_GradientBoosting_N16", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 7, "n_features": 16, "F1_dir": 0.5029, "F1_UP_FORT": 0.3143, "F1_DOWN_FORT": 0.3209, "train_start": "2000-11-10", "sampler": "SMOTETomek", "best_params": "{}", "features": ["NFCI_ret_5d__prod__CLX_Clorox_vol_20d", "NFCI_ret_5d__minus__NFCI_vol_20d", "Nikkei_Japan_vol_20d", "MKC_McCormick_ret_5d__minus__HangSeng_HK_ret_5d", "NFCI_ret_5d__minus__M_Macys_vol_20d", "LOW_Lowes_ret_20d", "CPB_CampbellSoup_ret_20d", "NFCI_ret_5d__prod__vix_zscore_10d", "heston_var_ev_h1__minus__M_Macys_vol_20d", "PAYX_Paychex_zscore_60d", "NFCI_ret_5d__minus__CLX_Clorox_vol_20d", "EWG_Germany_vol_20d", "MKC_McCormick_ret_5d__zrel__COST_ret_5d", "MKC_McCormick_ret_5d__minus__COST_ret_5d", "heston_var_ev_h1__minus__CLX_Clorox_vol_20d", "IBEX_Spain_ret_20d"], "is_new": false}, {"model_id": "v1_h7_NORMAL_RandomForest_N6", "algo": "RandomForest", "regime": "NORMAL", "horizon": 7, "n_features": 6, "F1_dir": 0.5038, "F1_UP_FORT": 0.3407, "F1_DOWN_FORT": 0.3243, "train_start": "2000-11-10", "sampler": "SMOTETomek", "best_params": "{}", "features": ["NFCI_ret_5d__prod__CLX_Clorox_vol_20d", "NFCI_ret_5d__minus__NFCI_vol_20d", "Nikkei_Japan_vol_20d", "MKC_McCormick_ret_5d__minus__HangSeng_HK_ret_5d", "NFCI_ret_5d__minus__M_Macys_vol_20d", "LOW_Lowes_ret_20d"], "is_new": false}, {"model_id": "v1_h7_NORMAL_RandomForest_N7", "algo": "RandomForest", "regime": "NORMAL", "horizon": 7, "n_features": 7, "F1_dir": 0.5089, "F1_UP_FORT": 0.3226, "F1_DOWN_FORT": 0.3568, "train_start": "2000-11-10", "sampler": "SMOTETomek", "best_params": "{}", "features": ["NFCI_ret_5d__prod__CLX_Clorox_vol_20d", "NFCI_ret_5d__minus__NFCI_vol_20d", "Nikkei_Japan_vol_20d", "MKC_McCormick_ret_5d__minus__HangSeng_HK_ret_5d", "NFCI_ret_5d__minus__M_Macys_vol_20d", "LOW_Lowes_ret_20d", "CPB_CampbellSoup_ret_20d"], "is_new": false}, {"model_id": "v1_h7_NORMAL_RandomForest_N8", "algo": "RandomForest", "regime": "NORMAL", "horizon": 7, "n_features": 8, "F1_dir": 0.5437, "F1_UP_FORT": 0.3262, "F1_DOWN_FORT": 0.3981, "train_start": "2000-11-10", "sampler": "SMOTETomek", "best_params": "{}", "features": ["NFCI_ret_5d__prod__CLX_Clorox_vol_20d", "NFCI_ret_5d__minus__NFCI_vol_20d", "Nikkei_Japan_vol_20d", "MKC_McCormick_ret_5d__minus__HangSeng_HK_ret_5d", "NFCI_ret_5d__minus__M_Macys_vol_20d", "LOW_Lowes_ret_20d", "CPB_CampbellSoup_ret_20d", "NFCI_ret_5d__prod__vix_zscore_10d"], "is_new": false}, {"model_id": "v1_h7_NORMAL_RandomForest_N9", "algo": "RandomForest", "regime": "NORMAL", "horizon": 7, "n_features": 9, "F1_dir": 0.5166, "F1_UP_FORT": 0.3501, "F1_DOWN_FORT": 0.3741, "train_start": "2000-11-10", "sampler": "SMOTETomek", "best_params": "{}", "features": ["NFCI_ret_5d__prod__CLX_Clorox_vol_20d", "NFCI_ret_5d__minus__NFCI_vol_20d", "Nikkei_Japan_vol_20d", "MKC_McCormick_ret_5d__minus__HangSeng_HK_ret_5d", "NFCI_ret_5d__minus__M_Macys_vol_20d", "LOW_Lowes_ret_20d", "CPB_CampbellSoup_ret_20d", "NFCI_ret_5d__prod__vix_zscore_10d", "heston_var_ev_h1__minus__M_Macys_vol_20d"], "is_new": false}, {"model_id": "v1_h7_NORMAL_RandomForest_N10", "algo": "RandomForest", "regime": "NORMAL", "horizon": 7, "n_features": 10, "F1_dir": 0.5408, "F1_UP_FORT": 0.385, "F1_DOWN_FORT": 0.3695, "train_start": "2000-11-10", "sampler": "SMOTETomek", "best_params": "{}", "features": ["NFCI_ret_5d__prod__CLX_Clorox_vol_20d", "NFCI_ret_5d__minus__NFCI_vol_20d", "Nikkei_Japan_vol_20d", "MKC_McCormick_ret_5d__minus__HangSeng_HK_ret_5d", "NFCI_ret_5d__minus__M_Macys_vol_20d", "LOW_Lowes_ret_20d", "CPB_CampbellSoup_ret_20d", "NFCI_ret_5d__prod__vix_zscore_10d", "heston_var_ev_h1__minus__M_Macys_vol_20d", "PAYX_Paychex_zscore_60d"], "is_new": false}, {"model_id": "v1_h7_NORMAL_RandomForest_N11", "algo": "RandomForest", "regime": "NORMAL", "horizon": 7, "n_features": 11, "F1_dir": 0.5293, "F1_UP_FORT": 0.3939, "F1_DOWN_FORT": 0.3616, "train_start": "2000-11-10", "sampler": "SMOTETomek", "best_params": "{}", "features": ["NFCI_ret_5d__prod__CLX_Clorox_vol_20d", "NFCI_ret_5d__minus__NFCI_vol_20d", "Nikkei_Japan_vol_20d", "MKC_McCormick_ret_5d__minus__HangSeng_HK_ret_5d", "NFCI_ret_5d__minus__M_Macys_vol_20d", "LOW_Lowes_ret_20d", "CPB_CampbellSoup_ret_20d", "NFCI_ret_5d__prod__vix_zscore_10d", "heston_var_ev_h1__minus__M_Macys_vol_20d", "PAYX_Paychex_zscore_60d", "NFCI_ret_5d__minus__CLX_Clorox_vol_20d"], "is_new": false}, {"model_id": "v1_h7_NORMAL_RandomForest_N12", "algo": "RandomForest", "regime": "NORMAL", "horizon": 7, "n_features": 12, "F1_dir": 0.5311, "F1_UP_FORT": 0.4069, "F1_DOWN_FORT": 0.3671, "train_start": "2000-11-10", "sampler": "SMOTETomek", "best_params": "{}", "features": ["NFCI_ret_5d__prod__CLX_Clorox_vol_20d", "NFCI_ret_5d__minus__NFCI_vol_20d", "Nikkei_Japan_vol_20d", "MKC_McCormick_ret_5d__minus__HangSeng_HK_ret_5d", "NFCI_ret_5d__minus__M_Macys_vol_20d", "LOW_Lowes_ret_20d", "CPB_CampbellSoup_ret_20d", "NFCI_ret_5d__prod__vix_zscore_10d", "heston_var_ev_h1__minus__M_Macys_vol_20d", "PAYX_Paychex_zscore_60d", "NFCI_ret_5d__minus__CLX_Clorox_vol_20d", "EWG_Germany_vol_20d"], "is_new": false}, {"model_id": "v1_h7_NORMAL_RandomForest_N13", "algo": "RandomForest", "regime": "NORMAL", "horizon": 7, "n_features": 13, "F1_dir": 0.5042, "F1_UP_FORT": 0.3805, "F1_DOWN_FORT": 0.3611, "train_start": "2000-11-10", "sampler": "SMOTETomek", "best_params": "{}", "features": ["NFCI_ret_5d__prod__CLX_Clorox_vol_20d", "NFCI_ret_5d__minus__NFCI_vol_20d", "Nikkei_Japan_vol_20d", "MKC_McCormick_ret_5d__minus__HangSeng_HK_ret_5d", "NFCI_ret_5d__minus__M_Macys_vol_20d", "LOW_Lowes_ret_20d", "CPB_CampbellSoup_ret_20d", "NFCI_ret_5d__prod__vix_zscore_10d", "heston_var_ev_h1__minus__M_Macys_vol_20d", "PAYX_Paychex_zscore_60d", "NFCI_ret_5d__minus__CLX_Clorox_vol_20d", "EWG_Germany_vol_20d", "MKC_McCormick_ret_5d__zrel__COST_ret_5d"], "is_new": false}, {"model_id": "v1_h7_NORMAL_RandomForest_N14", "algo": "RandomForest", "regime": "NORMAL", "horizon": 7, "n_features": 14, "F1_dir": 0.5097, "F1_UP_FORT": 0.3692, "F1_DOWN_FORT": 0.3804, "train_start": "2000-11-10", "sampler": "SMOTETomek", "best_params": "{}", "features": ["NFCI_ret_5d__prod__CLX_Clorox_vol_20d", "NFCI_ret_5d__minus__NFCI_vol_20d", "Nikkei_Japan_vol_20d", "MKC_McCormick_ret_5d__minus__HangSeng_HK_ret_5d", "NFCI_ret_5d__minus__M_Macys_vol_20d", "LOW_Lowes_ret_20d", "CPB_CampbellSoup_ret_20d", "NFCI_ret_5d__prod__vix_zscore_10d", "heston_var_ev_h1__minus__M_Macys_vol_20d", "PAYX_Paychex_zscore_60d", "NFCI_ret_5d__minus__CLX_Clorox_vol_20d", "EWG_Germany_vol_20d", "MKC_McCormick_ret_5d__zrel__COST_ret_5d", "MKC_McCormick_ret_5d__minus__COST_ret_5d"], "is_new": false}, {"model_id": "v1_h7_NORMAL_RandomForest_N15", "algo": "RandomForest", "regime": "NORMAL", "horizon": 7, "n_features": 15, "F1_dir": 0.5102, "F1_UP_FORT": 0.3509, "F1_DOWN_FORT": 0.3639, "train_start": "2000-11-10", "sampler": "SMOTETomek", "best_params": "{}", "features": ["NFCI_ret_5d__prod__CLX_Clorox_vol_20d", "NFCI_ret_5d__minus__NFCI_vol_20d", "Nikkei_Japan_vol_20d", "MKC_McCormick_ret_5d__minus__HangSeng_HK_ret_5d", "NFCI_ret_5d__minus__M_Macys_vol_20d", "LOW_Lowes_ret_20d", "CPB_CampbellSoup_ret_20d", "NFCI_ret_5d__prod__vix_zscore_10d", "heston_var_ev_h1__minus__M_Macys_vol_20d", "PAYX_Paychex_zscore_60d", "NFCI_ret_5d__minus__CLX_Clorox_vol_20d", "EWG_Germany_vol_20d", "MKC_McCormick_ret_5d__zrel__COST_ret_5d", "MKC_McCormick_ret_5d__minus__COST_ret_5d", "heston_var_ev_h1__minus__CLX_Clorox_vol_20d"], "is_new": false}, {"model_id": "v1_h7_NORMAL_RandomForest_N16", "algo": "RandomForest", "regime": "NORMAL", "horizon": 7, "n_features": 16, "F1_dir": 0.5271, "F1_UP_FORT": 0.3861, "F1_DOWN_FORT": 0.3573, "train_start": "2000-11-10", "sampler": "SMOTETomek", "best_params": "{}", "features": ["NFCI_ret_5d__prod__CLX_Clorox_vol_20d", "NFCI_ret_5d__minus__NFCI_vol_20d", "Nikkei_Japan_vol_20d", "MKC_McCormick_ret_5d__minus__HangSeng_HK_ret_5d", "NFCI_ret_5d__minus__M_Macys_vol_20d", "LOW_Lowes_ret_20d", "CPB_CampbellSoup_ret_20d", "NFCI_ret_5d__prod__vix_zscore_10d", "heston_var_ev_h1__minus__M_Macys_vol_20d", "PAYX_Paychex_zscore_60d", "NFCI_ret_5d__minus__CLX_Clorox_vol_20d", "EWG_Germany_vol_20d", "MKC_McCormick_ret_5d__zrel__COST_ret_5d", "MKC_McCormick_ret_5d__minus__COST_ret_5d", "heston_var_ev_h1__minus__CLX_Clorox_vol_20d", "IBEX_Spain_ret_20d"], "is_new": false}, {"model_id": "v1_h7_NORMAL_LogisticRegression_N6", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 7, "n_features": 6, "F1_dir": 0.5093, "F1_UP_FORT": 0.2884, "F1_DOWN_FORT": 0.335, "train_start": "2000-11-10", "sampler": "SMOTETomek", "best_params": "{}", "features": ["NFCI_ret_5d__prod__CLX_Clorox_vol_20d", "NFCI_ret_5d__minus__NFCI_vol_20d", "Nikkei_Japan_vol_20d", "MKC_McCormick_ret_5d__minus__HangSeng_HK_ret_5d", "NFCI_ret_5d__minus__M_Macys_vol_20d", "LOW_Lowes_ret_20d"], "is_new": false}, {"model_id": "v1_h7_NORMAL_LogisticRegression_N8", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 7, "n_features": 8, "F1_dir": 0.5351, "F1_UP_FORT": 0.3224, "F1_DOWN_FORT": 0.292, "train_start": "2000-11-10", "sampler": "SMOTETomek", "best_params": "{}", "features": ["NFCI_ret_5d__prod__CLX_Clorox_vol_20d", "NFCI_ret_5d__minus__NFCI_vol_20d", "Nikkei_Japan_vol_20d", "MKC_McCormick_ret_5d__minus__HangSeng_HK_ret_5d", "NFCI_ret_5d__minus__M_Macys_vol_20d", "LOW_Lowes_ret_20d", "CPB_CampbellSoup_ret_20d", "NFCI_ret_5d__prod__vix_zscore_10d"], "is_new": false}, {"model_id": "v1_h7_NORMAL_LogisticRegression_N9", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 7, "n_features": 9, "F1_dir": 0.512, "F1_UP_FORT": 0.2936, "F1_DOWN_FORT": 0.3383, "train_start": "2000-11-10", "sampler": "SMOTETomek", "best_params": "{}", "features": ["NFCI_ret_5d__prod__CLX_Clorox_vol_20d", "NFCI_ret_5d__minus__NFCI_vol_20d", "Nikkei_Japan_vol_20d", "MKC_McCormick_ret_5d__minus__HangSeng_HK_ret_5d", "NFCI_ret_5d__minus__M_Macys_vol_20d", "LOW_Lowes_ret_20d", "CPB_CampbellSoup_ret_20d", "NFCI_ret_5d__prod__vix_zscore_10d", "heston_var_ev_h1__minus__M_Macys_vol_20d"], "is_new": false}, {"model_id": "v1_h7_NORMAL_LogisticRegression_N10", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 7, "n_features": 10, "F1_dir": 0.5017, "F1_UP_FORT": 0.2957, "F1_DOWN_FORT": 0.3395, "train_start": "2000-11-10", "sampler": "SMOTETomek", "best_params": "{}", "features": ["NFCI_ret_5d__prod__CLX_Clorox_vol_20d", "NFCI_ret_5d__minus__NFCI_vol_20d", "Nikkei_Japan_vol_20d", "MKC_McCormick_ret_5d__minus__HangSeng_HK_ret_5d", "NFCI_ret_5d__minus__M_Macys_vol_20d", "LOW_Lowes_ret_20d", "CPB_CampbellSoup_ret_20d", "NFCI_ret_5d__prod__vix_zscore_10d", "heston_var_ev_h1__minus__M_Macys_vol_20d", "PAYX_Paychex_zscore_60d"], "is_new": false}, {"model_id": "v1_h7_NORMAL_LogisticRegression_N12", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 7, "n_features": 12, "F1_dir": 0.5046, "F1_UP_FORT": 0.3169, "F1_DOWN_FORT": 0.3354, "train_start": "2000-11-10", "sampler": "SMOTETomek", "best_params": "{}", "features": ["NFCI_ret_5d__prod__CLX_Clorox_vol_20d", "NFCI_ret_5d__minus__NFCI_vol_20d", "Nikkei_Japan_vol_20d", "MKC_McCormick_ret_5d__minus__HangSeng_HK_ret_5d", "NFCI_ret_5d__minus__M_Macys_vol_20d", "LOW_Lowes_ret_20d", "CPB_CampbellSoup_ret_20d", "NFCI_ret_5d__prod__vix_zscore_10d", "heston_var_ev_h1__minus__M_Macys_vol_20d", "PAYX_Paychex_zscore_60d", "NFCI_ret_5d__minus__CLX_Clorox_vol_20d", "EWG_Germany_vol_20d"], "is_new": false}, {"model_id": "v1_h7_NORMAL_LogisticRegression_N13", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 7, "n_features": 13, "F1_dir": 0.5192, "F1_UP_FORT": 0.3312, "F1_DOWN_FORT": 0.3534, "train_start": "2000-11-10", "sampler": "SMOTETomek", "best_params": "{}", "features": ["NFCI_ret_5d__prod__CLX_Clorox_vol_20d", "NFCI_ret_5d__minus__NFCI_vol_20d", "Nikkei_Japan_vol_20d", "MKC_McCormick_ret_5d__minus__HangSeng_HK_ret_5d", "NFCI_ret_5d__minus__M_Macys_vol_20d", "LOW_Lowes_ret_20d", "CPB_CampbellSoup_ret_20d", "NFCI_ret_5d__prod__vix_zscore_10d", "heston_var_ev_h1__minus__M_Macys_vol_20d", "PAYX_Paychex_zscore_60d", "NFCI_ret_5d__minus__CLX_Clorox_vol_20d", "EWG_Germany_vol_20d", "MKC_McCormick_ret_5d__zrel__COST_ret_5d"], "is_new": false}, {"model_id": "v1_h7_NORMAL_LogisticRegression_N14", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 7, "n_features": 14, "F1_dir": 0.5278, "F1_UP_FORT": 0.3716, "F1_DOWN_FORT": 0.352, "train_start": "2000-11-10", "sampler": "SMOTETomek", "best_params": "{}", "features": ["NFCI_ret_5d__prod__CLX_Clorox_vol_20d", "NFCI_ret_5d__minus__NFCI_vol_20d", "Nikkei_Japan_vol_20d", "MKC_McCormick_ret_5d__minus__HangSeng_HK_ret_5d", "NFCI_ret_5d__minus__M_Macys_vol_20d", "LOW_Lowes_ret_20d", "CPB_CampbellSoup_ret_20d", "NFCI_ret_5d__prod__vix_zscore_10d", "heston_var_ev_h1__minus__M_Macys_vol_20d", "PAYX_Paychex_zscore_60d", "NFCI_ret_5d__minus__CLX_Clorox_vol_20d", "EWG_Germany_vol_20d", "MKC_McCormick_ret_5d__zrel__COST_ret_5d", "MKC_McCormick_ret_5d__minus__COST_ret_5d"], "is_new": false}, {"model_id": "v1_h7_NORMAL_LogisticRegression_N15", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 7, "n_features": 15, "F1_dir": 0.5249, "F1_UP_FORT": 0.3716, "F1_DOWN_FORT": 0.3499, "train_start": "2000-11-10", "sampler": "SMOTETomek", "best_params": "{}", "features": ["NFCI_ret_5d__prod__CLX_Clorox_vol_20d", "NFCI_ret_5d__minus__NFCI_vol_20d", "Nikkei_Japan_vol_20d", "MKC_McCormick_ret_5d__minus__HangSeng_HK_ret_5d", "NFCI_ret_5d__minus__M_Macys_vol_20d", "LOW_Lowes_ret_20d", "CPB_CampbellSoup_ret_20d", "NFCI_ret_5d__prod__vix_zscore_10d", "heston_var_ev_h1__minus__M_Macys_vol_20d", "PAYX_Paychex_zscore_60d", "NFCI_ret_5d__minus__CLX_Clorox_vol_20d", "EWG_Germany_vol_20d", "MKC_McCormick_ret_5d__zrel__COST_ret_5d", "MKC_McCormick_ret_5d__minus__COST_ret_5d", "heston_var_ev_h1__minus__CLX_Clorox_vol_20d"], "is_new": false}, {"model_id": "v1_h7_NORMAL_LogisticRegression_N16", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 7, "n_features": 16, "F1_dir": 0.5265, "F1_UP_FORT": 0.3771, "F1_DOWN_FORT": 0.3496, "train_start": "2000-11-10", "sampler": "SMOTETomek", "best_params": "{}", "features": ["NFCI_ret_5d__prod__CLX_Clorox_vol_20d", "NFCI_ret_5d__minus__NFCI_vol_20d", "Nikkei_Japan_vol_20d", "MKC_McCormick_ret_5d__minus__HangSeng_HK_ret_5d", "NFCI_ret_5d__minus__M_Macys_vol_20d", "LOW_Lowes_ret_20d", "CPB_CampbellSoup_ret_20d", "NFCI_ret_5d__prod__vix_zscore_10d", "heston_var_ev_h1__minus__M_Macys_vol_20d", "PAYX_Paychex_zscore_60d", "NFCI_ret_5d__minus__CLX_Clorox_vol_20d", "EWG_Germany_vol_20d", "MKC_McCormick_ret_5d__zrel__COST_ret_5d", "MKC_McCormick_ret_5d__minus__COST_ret_5d", "heston_var_ev_h1__minus__CLX_Clorox_vol_20d", "IBEX_Spain_ret_20d"], "is_new": false}, {"model_id": "v1_h7_NORMAL_RandomForest_Optuna_N8", "algo": "RandomForest", "regime": "NORMAL", "horizon": 7, "n_features": 8, "F1_dir": 0.5498, "F1_UP_FORT": 0.3375, "F1_DOWN_FORT": 0.3731, "train_start": "2000-11-10", "sampler": "SMOTETomek", "best_params": "{}", "features": ["NFCI_ret_5d__prod__CLX_Clorox_vol_20d", "NFCI_ret_5d__minus__NFCI_vol_20d", "Nikkei_Japan_vol_20d", "MKC_McCormick_ret_5d__minus__HangSeng_HK_ret_5d", "NFCI_ret_5d__minus__M_Macys_vol_20d", "LOW_Lowes_ret_20d", "CPB_CampbellSoup_ret_20d", "NFCI_ret_5d__prod__vix_zscore_10d"], "is_new": false}, {"model_id": "v1_h7_NORMAL_RandomForest_OptunaCal_N8", "algo": "RandomForestCal", "regime": "NORMAL", "horizon": 7, "n_features": 8, "F1_dir": 0.5151, "F1_UP_FORT": 0.3424, "F1_DOWN_FORT": 0.3469, "train_start": "2000-11-10", "sampler": "SMOTETomek", "best_params": "{}", "features": ["NFCI_ret_5d__prod__CLX_Clorox_vol_20d", "NFCI_ret_5d__minus__NFCI_vol_20d", "Nikkei_Japan_vol_20d", "MKC_McCormick_ret_5d__minus__HangSeng_HK_ret_5d", "NFCI_ret_5d__minus__M_Macys_vol_20d", "LOW_Lowes_ret_20d", "CPB_CampbellSoup_ret_20d", "NFCI_ret_5d__prod__vix_zscore_10d"], "is_new": false}, {"model_id": "v1_h7_STRESS_LogisticRegression_N5", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 7, "n_features": 5, "F1_dir": 0.5167, "F1_UP_FORT": 0.1042, "F1_DOWN_FORT": 0.3866, "train_start": "2001-02-07", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["SBUX_ret_5d__div__heston_var_ev_h3", "vix_momentum_2d__minus__US20Y_Rate_ret_20d", "DE_Deere_ret_5d__minus__hmm_p_stress", "US3M_Rate_vol_20d__prod__hmm_p_stress", "DE_Deere_ret_5d__ret5x__hmm_p_stress"], "is_new": false}, {"model_id": "v1_h7_STRESS_LogisticRegression_N6", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 7, "n_features": 6, "F1_dir": 0.5311, "F1_UP_FORT": 0.219, "F1_DOWN_FORT": 0.3498, "train_start": "2001-02-07", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["SBUX_ret_5d__div__heston_var_ev_h3", "vix_momentum_2d__minus__US20Y_Rate_ret_20d", "DE_Deere_ret_5d__minus__hmm_p_stress", "US3M_Rate_vol_20d__prod__hmm_p_stress", "DE_Deere_ret_5d__ret5x__hmm_p_stress", "ROST_RossStores_ret_5d__minus__DE_Deere_ret_5d"], "is_new": false}, {"model_id": "v1_h7_STRESS_LogisticRegression_N7", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 7, "n_features": 7, "F1_dir": 0.5376, "F1_UP_FORT": 0.219, "F1_DOWN_FORT": 0.37, "train_start": "2001-02-07", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["SBUX_ret_5d__div__heston_var_ev_h3", "vix_momentum_2d__minus__US20Y_Rate_ret_20d", "DE_Deere_ret_5d__minus__hmm_p_stress", "US3M_Rate_vol_20d__prod__hmm_p_stress", "DE_Deere_ret_5d__ret5x__hmm_p_stress", "ROST_RossStores_ret_5d__minus__DE_Deere_ret_5d", "HUM_Humana_zscore_60d__div__US20Y_Rate_ret_20d"], "is_new": false}, {"model_id": "v1_h7_STRESS_LogisticRegression_N8", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 7, "n_features": 8, "F1_dir": 0.5056, "F1_UP_FORT": 0.1818, "F1_DOWN_FORT": 0.3041, "train_start": "2001-02-07", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["SBUX_ret_5d__div__heston_var_ev_h3", "vix_momentum_2d__minus__US20Y_Rate_ret_20d", "DE_Deere_ret_5d__minus__hmm_p_stress", "US3M_Rate_vol_20d__prod__hmm_p_stress", "DE_Deere_ret_5d__ret5x__hmm_p_stress", "ROST_RossStores_ret_5d__minus__DE_Deere_ret_5d", "HUM_Humana_zscore_60d__div__US20Y_Rate_ret_20d", "DHR_vol_20d"], "is_new": false}, {"model_id": "v1_h7_STRESS_LogisticRegression_N9", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 7, "n_features": 9, "F1_dir": 0.5205, "F1_UP_FORT": 0.1884, "F1_DOWN_FORT": 0.3421, "train_start": "2001-02-07", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["SBUX_ret_5d__div__heston_var_ev_h3", "vix_momentum_2d__minus__US20Y_Rate_ret_20d", "DE_Deere_ret_5d__minus__hmm_p_stress", "US3M_Rate_vol_20d__prod__hmm_p_stress", "DE_Deere_ret_5d__ret5x__hmm_p_stress", "ROST_RossStores_ret_5d__minus__DE_Deere_ret_5d", "HUM_Humana_zscore_60d__div__US20Y_Rate_ret_20d", "DHR_vol_20d", "AMD_ret_1d"], "is_new": false}, {"model_id": "v1_h7_STRESS_LogisticRegression_N10", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 7, "n_features": 10, "F1_dir": 0.5225, "F1_UP_FORT": 0.1765, "F1_DOWN_FORT": 0.3482, "train_start": "2001-02-07", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["SBUX_ret_5d__div__heston_var_ev_h3", "vix_momentum_2d__minus__US20Y_Rate_ret_20d", "DE_Deere_ret_5d__minus__hmm_p_stress", "US3M_Rate_vol_20d__prod__hmm_p_stress", "DE_Deere_ret_5d__ret5x__hmm_p_stress", "ROST_RossStores_ret_5d__minus__DE_Deere_ret_5d", "HUM_Humana_zscore_60d__div__US20Y_Rate_ret_20d", "DHR_vol_20d", "AMD_ret_1d", "PPL_PPL_ret_1d"], "is_new": false}, {"model_id": "v1_h7_STRESS_LogisticRegression_N11", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 7, "n_features": 11, "F1_dir": 0.5161, "F1_UP_FORT": 0.1871, "F1_DOWN_FORT": 0.3455, "train_start": "2001-02-07", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["SBUX_ret_5d__div__heston_var_ev_h3", "vix_momentum_2d__minus__US20Y_Rate_ret_20d", "DE_Deere_ret_5d__minus__hmm_p_stress", "US3M_Rate_vol_20d__prod__hmm_p_stress", "DE_Deere_ret_5d__ret5x__hmm_p_stress", "ROST_RossStores_ret_5d__minus__DE_Deere_ret_5d", "HUM_Humana_zscore_60d__div__US20Y_Rate_ret_20d", "DHR_vol_20d", "AMD_ret_1d", "PPL_PPL_ret_1d", "US3M_Rate_vol_20d__ret5x__DE_Deere_ret_5d"], "is_new": false}, {"model_id": "v1_h7_STRESS_LogisticRegression_N12", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 7, "n_features": 12, "F1_dir": 0.5183, "F1_UP_FORT": 0.1871, "F1_DOWN_FORT": 0.338, "train_start": "2001-02-07", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["SBUX_ret_5d__div__heston_var_ev_h3", "vix_momentum_2d__minus__US20Y_Rate_ret_20d", "DE_Deere_ret_5d__minus__hmm_p_stress", "US3M_Rate_vol_20d__prod__hmm_p_stress", "DE_Deere_ret_5d__ret5x__hmm_p_stress", "ROST_RossStores_ret_5d__minus__DE_Deere_ret_5d", "HUM_Humana_zscore_60d__div__US20Y_Rate_ret_20d", "DHR_vol_20d", "AMD_ret_1d", "PPL_PPL_ret_1d", "US3M_Rate_vol_20d__ret5x__DE_Deere_ret_5d", "SBUX_ret_5d__minus__hmm_p_stress"], "is_new": false}, {"model_id": "v1_h7_STRESS_LogisticRegression_N13", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 7, "n_features": 13, "F1_dir": 0.5011, "F1_UP_FORT": 0.1831, "F1_DOWN_FORT": 0.3585, "train_start": "2001-02-07", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["SBUX_ret_5d__div__heston_var_ev_h3", "vix_momentum_2d__minus__US20Y_Rate_ret_20d", "DE_Deere_ret_5d__minus__hmm_p_stress", "US3M_Rate_vol_20d__prod__hmm_p_stress", "DE_Deere_ret_5d__ret5x__hmm_p_stress", "ROST_RossStores_ret_5d__minus__DE_Deere_ret_5d", "HUM_Humana_zscore_60d__div__US20Y_Rate_ret_20d", "DHR_vol_20d", "AMD_ret_1d", "PPL_PPL_ret_1d", "US3M_Rate_vol_20d__ret5x__DE_Deere_ret_5d", "SBUX_ret_5d__minus__hmm_p_stress", "STLFSI4_zscore_60d__div__FedFunds_zscore_60d"], "is_new": false}, {"model_id": "v1_h7_GLOBAL_XGBoost_N5", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 7, "n_features": 5, "F1_dir": 0.5545, "F1_UP_FORT": 0.3606, "F1_DOWN_FORT": 0.3003, "train_start": "2001-02-07", "sampler": "SMOTE", "best_params": "{}", "features": ["NFCI_ret_5d__div__VRP", "NFCI_ret_5d__ret5x__NFCI_ret_20d", "CPB_CampbellSoup_vol_20d", "CLX_Clorox_vol_20d", "US3Y_Rate_ret_5d"], "is_new": false}, {"model_id": "v1_h7_GLOBAL_XGBoost_N6", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 7, "n_features": 6, "F1_dir": 0.5495, "F1_UP_FORT": 0.3584, "F1_DOWN_FORT": 0.3422, "train_start": "2001-02-07", "sampler": "SMOTE", "best_params": "{}", "features": ["NFCI_ret_5d__div__VRP", "NFCI_ret_5d__ret5x__NFCI_ret_20d", "CPB_CampbellSoup_vol_20d", "CLX_Clorox_vol_20d", "US3Y_Rate_ret_5d", "EWA_Australia_zscore_60d"], "is_new": false}, {"model_id": "v1_h7_GLOBAL_XGBoost_N7", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 7, "n_features": 7, "F1_dir": 0.5768, "F1_UP_FORT": 0.3626, "F1_DOWN_FORT": 0.3781, "train_start": "2001-02-07", "sampler": "SMOTE", "best_params": "{}", "features": ["NFCI_ret_5d__div__VRP", "NFCI_ret_5d__ret5x__NFCI_ret_20d", "CPB_CampbellSoup_vol_20d", "CLX_Clorox_vol_20d", "US3Y_Rate_ret_5d", "EWA_Australia_zscore_60d", "XLV_Health_zscore_60d"], "is_new": false}, {"model_id": "v1_h7_GLOBAL_XGBoost_N8", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 7, "n_features": 8, "F1_dir": 0.5924, "F1_UP_FORT": 0.3631, "F1_DOWN_FORT": 0.4026, "train_start": "2001-02-07", "sampler": "SMOTE", "best_params": "{}", "features": ["NFCI_ret_5d__div__VRP", "NFCI_ret_5d__ret5x__NFCI_ret_20d", "CPB_CampbellSoup_vol_20d", "CLX_Clorox_vol_20d", "US3Y_Rate_ret_5d", "EWA_Australia_zscore_60d", "XLV_Health_zscore_60d", "CTAS_Cintas_vol_20d"], "is_new": false}, {"model_id": "v1_h7_GLOBAL_XGBoost_N9", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 7, "n_features": 9, "F1_dir": 0.5937, "F1_UP_FORT": 0.3477, "F1_DOWN_FORT": 0.4132, "train_start": "2001-02-07", "sampler": "SMOTE", "best_params": "{}", "features": ["NFCI_ret_5d__div__VRP", "NFCI_ret_5d__ret5x__NFCI_ret_20d", "CPB_CampbellSoup_vol_20d", "CLX_Clorox_vol_20d", "US3Y_Rate_ret_5d", "EWA_Australia_zscore_60d", "XLV_Health_zscore_60d", "CTAS_Cintas_vol_20d", "GILD_Gilead_ret_20d"], "is_new": false}, {"model_id": "v1_h7_GLOBAL_XGBoost_N10", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 7, "n_features": 10, "F1_dir": 0.5842, "F1_UP_FORT": 0.337, "F1_DOWN_FORT": 0.4023, "train_start": "2001-02-07", "sampler": "SMOTE", "best_params": "{}", "features": ["NFCI_ret_5d__div__VRP", "NFCI_ret_5d__ret5x__NFCI_ret_20d", "CPB_CampbellSoup_vol_20d", "CLX_Clorox_vol_20d", "US3Y_Rate_ret_5d", "EWA_Australia_zscore_60d", "XLV_Health_zscore_60d", "CTAS_Cintas_vol_20d", "GILD_Gilead_ret_20d", "heston_xi__prod__kalman_filtered"], "is_new": false}, {"model_id": "v1_h7_GLOBAL_XGBoost_N11", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 7, "n_features": 11, "F1_dir": 0.5918, "F1_UP_FORT": 0.3733, "F1_DOWN_FORT": 0.4148, "train_start": "2001-02-07", "sampler": "SMOTE", "best_params": "{}", "features": ["NFCI_ret_5d__div__VRP", "NFCI_ret_5d__ret5x__NFCI_ret_20d", "CPB_CampbellSoup_vol_20d", "CLX_Clorox_vol_20d", "US3Y_Rate_ret_5d", "EWA_Australia_zscore_60d", "XLV_Health_zscore_60d", "CTAS_Cintas_vol_20d", "GILD_Gilead_ret_20d", "heston_xi__prod__kalman_filtered", "EWS_Singapore_ret_5d"], "is_new": false}, {"model_id": "v1_h7_GLOBAL_XGBoost_N12", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 7, "n_features": 12, "F1_dir": 0.611, "F1_UP_FORT": 0.4178, "F1_DOWN_FORT": 0.4434, "train_start": "2001-02-07", "sampler": "SMOTE", "best_params": "{}", "features": ["NFCI_ret_5d__div__VRP", "NFCI_ret_5d__ret5x__NFCI_ret_20d", "CPB_CampbellSoup_vol_20d", "CLX_Clorox_vol_20d", "US3Y_Rate_ret_5d", "EWA_Australia_zscore_60d", "XLV_Health_zscore_60d", "CTAS_Cintas_vol_20d", "GILD_Gilead_ret_20d", "heston_xi__prod__kalman_filtered", "EWS_Singapore_ret_5d", "VVIX_vol_20d__zrel__kalman_filtered"], "is_new": false}, {"model_id": "v1_h7_GLOBAL_XGBoost_N13", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 7, "n_features": 13, "F1_dir": 0.5933, "F1_UP_FORT": 0.3748, "F1_DOWN_FORT": 0.4378, "train_start": "2001-02-07", "sampler": "SMOTE", "best_params": "{}", "features": ["NFCI_ret_5d__div__VRP", "NFCI_ret_5d__ret5x__NFCI_ret_20d", "CPB_CampbellSoup_vol_20d", "CLX_Clorox_vol_20d", "US3Y_Rate_ret_5d", "EWA_Australia_zscore_60d", "XLV_Health_zscore_60d", "CTAS_Cintas_vol_20d", "GILD_Gilead_ret_20d", "heston_xi__prod__kalman_filtered", "EWS_Singapore_ret_5d", "VVIX_vol_20d__zrel__kalman_filtered", "Nikkei_Japan_vol_20d"], "is_new": false}, {"model_id": "v1_h7_GLOBAL_XGBoost_N14", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 7, "n_features": 14, "F1_dir": 0.5996, "F1_UP_FORT": 0.3833, "F1_DOWN_FORT": 0.4561, "train_start": "2001-02-07", "sampler": "SMOTE", "best_params": "{}", "features": ["NFCI_ret_5d__div__VRP", "NFCI_ret_5d__ret5x__NFCI_ret_20d", "CPB_CampbellSoup_vol_20d", "CLX_Clorox_vol_20d", "US3Y_Rate_ret_5d", "EWA_Australia_zscore_60d", "XLV_Health_zscore_60d", "CTAS_Cintas_vol_20d", "GILD_Gilead_ret_20d", "heston_xi__prod__kalman_filtered", "EWS_Singapore_ret_5d", "VVIX_vol_20d__zrel__kalman_filtered", "Nikkei_Japan_vol_20d", "SBUX_ret_5d"], "is_new": false}, {"model_id": "v1_h7_GLOBAL_XGBoost_N15", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 7, "n_features": 15, "F1_dir": 0.6091, "F1_UP_FORT": 0.3931, "F1_DOWN_FORT": 0.4857, "train_start": "2001-02-07", "sampler": "SMOTE", "best_params": "{}", "features": ["NFCI_ret_5d__div__VRP", "NFCI_ret_5d__ret5x__NFCI_ret_20d", "CPB_CampbellSoup_vol_20d", "CLX_Clorox_vol_20d", "US3Y_Rate_ret_5d", "EWA_Australia_zscore_60d", "XLV_Health_zscore_60d", "CTAS_Cintas_vol_20d", "GILD_Gilead_ret_20d", "heston_xi__prod__kalman_filtered", "EWS_Singapore_ret_5d", "VVIX_vol_20d__zrel__kalman_filtered", "Nikkei_Japan_vol_20d", "SBUX_ret_5d", "VVIX_vol_20d__prod__kalman_filtered"], "is_new": false}, {"model_id": "v1_h7_GLOBAL_XGBoost_N16", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 7, "n_features": 16, "F1_dir": 0.6083, "F1_UP_FORT": 0.3926, "F1_DOWN_FORT": 0.4767, "train_start": "2001-02-07", "sampler": "SMOTE", "best_params": "{}", "features": ["NFCI_ret_5d__div__VRP", "NFCI_ret_5d__ret5x__NFCI_ret_20d", "CPB_CampbellSoup_vol_20d", "CLX_Clorox_vol_20d", "US3Y_Rate_ret_5d", "EWA_Australia_zscore_60d", "XLV_Health_zscore_60d", "CTAS_Cintas_vol_20d", "GILD_Gilead_ret_20d", "heston_xi__prod__kalman_filtered", "EWS_Singapore_ret_5d", "VVIX_vol_20d__zrel__kalman_filtered", "Nikkei_Japan_vol_20d", "SBUX_ret_5d", "VVIX_vol_20d__prod__kalman_filtered", "PAYX_Paychex_ret_20d"], "is_new": false}, {"model_id": "v1_h7_GLOBAL_XGBoost_N17", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 7, "n_features": 17, "F1_dir": 0.6261, "F1_UP_FORT": 0.4375, "F1_DOWN_FORT": 0.4741, "train_start": "2001-02-07", "sampler": "SMOTE", "best_params": "{}", "features": ["NFCI_ret_5d__div__VRP", "NFCI_ret_5d__ret5x__NFCI_ret_20d", "CPB_CampbellSoup_vol_20d", "CLX_Clorox_vol_20d", "US3Y_Rate_ret_5d", "EWA_Australia_zscore_60d", "XLV_Health_zscore_60d", "CTAS_Cintas_vol_20d", "GILD_Gilead_ret_20d", "heston_xi__prod__kalman_filtered", "EWS_Singapore_ret_5d", "VVIX_vol_20d__zrel__kalman_filtered", "Nikkei_Japan_vol_20d", "SBUX_ret_5d", "VVIX_vol_20d__prod__kalman_filtered", "PAYX_Paychex_ret_20d", "NFCI_ret_5d__div__NFCI_vol_20d"], "is_new": false}, {"model_id": "v1_h7_GLOBAL_LightGBM_N5", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 7, "n_features": 5, "F1_dir": 0.5226, "F1_UP_FORT": 0.3046, "F1_DOWN_FORT": 0.2885, "train_start": "2001-02-07", "sampler": "SMOTE", "best_params": "{}", "features": ["NFCI_ret_5d__div__VRP", "NFCI_ret_5d__ret5x__NFCI_ret_20d", "CPB_CampbellSoup_vol_20d", "CLX_Clorox_vol_20d", "US3Y_Rate_ret_5d"], "is_new": false}, {"model_id": "v1_h7_GLOBAL_LightGBM_N6", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 7, "n_features": 6, "F1_dir": 0.5761, "F1_UP_FORT": 0.3639, "F1_DOWN_FORT": 0.3437, "train_start": "2001-02-07", "sampler": "SMOTE", "best_params": "{}", "features": ["NFCI_ret_5d__div__VRP", "NFCI_ret_5d__ret5x__NFCI_ret_20d", "CPB_CampbellSoup_vol_20d", "CLX_Clorox_vol_20d", "US3Y_Rate_ret_5d", "EWA_Australia_zscore_60d"], "is_new": false}, {"model_id": "v1_h7_GLOBAL_LightGBM_N7", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 7, "n_features": 7, "F1_dir": 0.5656, "F1_UP_FORT": 0.3372, "F1_DOWN_FORT": 0.3731, "train_start": "2001-02-07", "sampler": "SMOTE", "best_params": "{}", "features": ["NFCI_ret_5d__div__VRP", "NFCI_ret_5d__ret5x__NFCI_ret_20d", "CPB_CampbellSoup_vol_20d", "CLX_Clorox_vol_20d", "US3Y_Rate_ret_5d", "EWA_Australia_zscore_60d", "XLV_Health_zscore_60d"], "is_new": false}, {"model_id": "v1_h7_GLOBAL_LightGBM_N8", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 7, "n_features": 8, "F1_dir": 0.602, "F1_UP_FORT": 0.3538, "F1_DOWN_FORT": 0.4185, "train_start": "2001-02-07", "sampler": "SMOTE", "best_params": "{}", "features": ["NFCI_ret_5d__div__VRP", "NFCI_ret_5d__ret5x__NFCI_ret_20d", "CPB_CampbellSoup_vol_20d", "CLX_Clorox_vol_20d", "US3Y_Rate_ret_5d", "EWA_Australia_zscore_60d", "XLV_Health_zscore_60d", "CTAS_Cintas_vol_20d"], "is_new": false}, {"model_id": "v1_h7_GLOBAL_LightGBM_N9", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 7, "n_features": 9, "F1_dir": 0.5962, "F1_UP_FORT": 0.3468, "F1_DOWN_FORT": 0.4016, "train_start": "2001-02-07", "sampler": "SMOTE", "best_params": "{}", "features": ["NFCI_ret_5d__div__VRP", "NFCI_ret_5d__ret5x__NFCI_ret_20d", "CPB_CampbellSoup_vol_20d", "CLX_Clorox_vol_20d", "US3Y_Rate_ret_5d", "EWA_Australia_zscore_60d", "XLV_Health_zscore_60d", "CTAS_Cintas_vol_20d", "GILD_Gilead_ret_20d"], "is_new": false}, {"model_id": "v1_h7_GLOBAL_LightGBM_N10", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 7, "n_features": 10, "F1_dir": 0.5929, "F1_UP_FORT": 0.3666, "F1_DOWN_FORT": 0.4204, "train_start": "2001-02-07", "sampler": "SMOTE", "best_params": "{}", "features": ["NFCI_ret_5d__div__VRP", "NFCI_ret_5d__ret5x__NFCI_ret_20d", "CPB_CampbellSoup_vol_20d", "CLX_Clorox_vol_20d", "US3Y_Rate_ret_5d", "EWA_Australia_zscore_60d", "XLV_Health_zscore_60d", "CTAS_Cintas_vol_20d", "GILD_Gilead_ret_20d", "heston_xi__prod__kalman_filtered"], "is_new": false}, {"model_id": "v1_h7_GLOBAL_LightGBM_N11", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 7, "n_features": 11, "F1_dir": 0.6001, "F1_UP_FORT": 0.3776, "F1_DOWN_FORT": 0.4264, "train_start": "2001-02-07", "sampler": "SMOTE", "best_params": "{}", "features": ["NFCI_ret_5d__div__VRP", "NFCI_ret_5d__ret5x__NFCI_ret_20d", "CPB_CampbellSoup_vol_20d", "CLX_Clorox_vol_20d", "US3Y_Rate_ret_5d", "EWA_Australia_zscore_60d", "XLV_Health_zscore_60d", "CTAS_Cintas_vol_20d", "GILD_Gilead_ret_20d", "heston_xi__prod__kalman_filtered", "EWS_Singapore_ret_5d"], "is_new": false}, {"model_id": "v1_h7_GLOBAL_LightGBM_N12", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 7, "n_features": 12, "F1_dir": 0.6191, "F1_UP_FORT": 0.4, "F1_DOWN_FORT": 0.4195, "train_start": "2001-02-07", "sampler": "SMOTE", "best_params": "{}", "features": ["NFCI_ret_5d__div__VRP", "NFCI_ret_5d__ret5x__NFCI_ret_20d", "CPB_CampbellSoup_vol_20d", "CLX_Clorox_vol_20d", "US3Y_Rate_ret_5d", "EWA_Australia_zscore_60d", "XLV_Health_zscore_60d", "CTAS_Cintas_vol_20d", "GILD_Gilead_ret_20d", "heston_xi__prod__kalman_filtered", "EWS_Singapore_ret_5d", "VVIX_vol_20d__zrel__kalman_filtered"], "is_new": false}, {"model_id": "v1_h7_GLOBAL_LightGBM_N13", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 7, "n_features": 13, "F1_dir": 0.5896, "F1_UP_FORT": 0.3737, "F1_DOWN_FORT": 0.4305, "train_start": "2001-02-07", "sampler": "SMOTE", "best_params": "{}", "features": ["NFCI_ret_5d__div__VRP", "NFCI_ret_5d__ret5x__NFCI_ret_20d", "CPB_CampbellSoup_vol_20d", "CLX_Clorox_vol_20d", "US3Y_Rate_ret_5d", "EWA_Australia_zscore_60d", "XLV_Health_zscore_60d", "CTAS_Cintas_vol_20d", "GILD_Gilead_ret_20d", "heston_xi__prod__kalman_filtered", "EWS_Singapore_ret_5d", "VVIX_vol_20d__zrel__kalman_filtered", "Nikkei_Japan_vol_20d"], "is_new": false}, {"model_id": "v1_h7_GLOBAL_LightGBM_N14", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 7, "n_features": 14, "F1_dir": 0.6199, "F1_UP_FORT": 0.3733, "F1_DOWN_FORT": 0.4641, "train_start": "2001-02-07", "sampler": "SMOTE", "best_params": "{}", "features": ["NFCI_ret_5d__div__VRP", "NFCI_ret_5d__ret5x__NFCI_ret_20d", "CPB_CampbellSoup_vol_20d", "CLX_Clorox_vol_20d", "US3Y_Rate_ret_5d", "EWA_Australia_zscore_60d", "XLV_Health_zscore_60d", "CTAS_Cintas_vol_20d", "GILD_Gilead_ret_20d", "heston_xi__prod__kalman_filtered", "EWS_Singapore_ret_5d", "VVIX_vol_20d__zrel__kalman_filtered", "Nikkei_Japan_vol_20d", "SBUX_ret_5d"], "is_new": false}, {"model_id": "v1_h7_GLOBAL_LightGBM_N15", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 7, "n_features": 15, "F1_dir": 0.6262, "F1_UP_FORT": 0.4107, "F1_DOWN_FORT": 0.4816, "train_start": "2001-02-07", "sampler": "SMOTE", "best_params": "{}", "features": ["NFCI_ret_5d__div__VRP", "NFCI_ret_5d__ret5x__NFCI_ret_20d", "CPB_CampbellSoup_vol_20d", "CLX_Clorox_vol_20d", "US3Y_Rate_ret_5d", "EWA_Australia_zscore_60d", "XLV_Health_zscore_60d", "CTAS_Cintas_vol_20d", "GILD_Gilead_ret_20d", "heston_xi__prod__kalman_filtered", "EWS_Singapore_ret_5d", "VVIX_vol_20d__zrel__kalman_filtered", "Nikkei_Japan_vol_20d", "SBUX_ret_5d", "VVIX_vol_20d__prod__kalman_filtered"], "is_new": false}, {"model_id": "v1_h7_GLOBAL_LightGBM_N16", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 7, "n_features": 16, "F1_dir": 0.621, "F1_UP_FORT": 0.3894, "F1_DOWN_FORT": 0.4826, "train_start": "2001-02-07", "sampler": "SMOTE", "best_params": "{}", "features": ["NFCI_ret_5d__div__VRP", "NFCI_ret_5d__ret5x__NFCI_ret_20d", "CPB_CampbellSoup_vol_20d", "CLX_Clorox_vol_20d", "US3Y_Rate_ret_5d", "EWA_Australia_zscore_60d", "XLV_Health_zscore_60d", "CTAS_Cintas_vol_20d", "GILD_Gilead_ret_20d", "heston_xi__prod__kalman_filtered", "EWS_Singapore_ret_5d", "VVIX_vol_20d__zrel__kalman_filtered", "Nikkei_Japan_vol_20d", "SBUX_ret_5d", "VVIX_vol_20d__prod__kalman_filtered", "PAYX_Paychex_ret_20d"], "is_new": false}, {"model_id": "v1_h7_GLOBAL_LightGBM_N17", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 7, "n_features": 17, "F1_dir": 0.6348, "F1_UP_FORT": 0.4307, "F1_DOWN_FORT": 0.4733, "train_start": "2001-02-07", "sampler": "SMOTE", "best_params": "{}", "features": ["NFCI_ret_5d__div__VRP", "NFCI_ret_5d__ret5x__NFCI_ret_20d", "CPB_CampbellSoup_vol_20d", "CLX_Clorox_vol_20d", "US3Y_Rate_ret_5d", "EWA_Australia_zscore_60d", "XLV_Health_zscore_60d", "CTAS_Cintas_vol_20d", "GILD_Gilead_ret_20d", "heston_xi__prod__kalman_filtered", "EWS_Singapore_ret_5d", "VVIX_vol_20d__zrel__kalman_filtered", "Nikkei_Japan_vol_20d", "SBUX_ret_5d", "VVIX_vol_20d__prod__kalman_filtered", "PAYX_Paychex_ret_20d", "NFCI_ret_5d__div__NFCI_vol_20d"], "is_new": false}, {"model_id": "v1_h7_GLOBAL_GradientBoosting_N5", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 7, "n_features": 5, "F1_dir": 0.5485, "F1_UP_FORT": 0.3803, "F1_DOWN_FORT": 0.2747, "train_start": "2001-02-07", "sampler": "SMOTE", "best_params": "{}", "features": ["NFCI_ret_5d__div__VRP", "NFCI_ret_5d__ret5x__NFCI_ret_20d", "CPB_CampbellSoup_vol_20d", "CLX_Clorox_vol_20d", "US3Y_Rate_ret_5d"], "is_new": false}, {"model_id": "v1_h7_GLOBAL_GradientBoosting_N6", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 7, "n_features": 6, "F1_dir": 0.567, "F1_UP_FORT": 0.3243, "F1_DOWN_FORT": 0.3544, "train_start": "2001-02-07", "sampler": "SMOTE", "best_params": "{}", "features": ["NFCI_ret_5d__div__VRP", "NFCI_ret_5d__ret5x__NFCI_ret_20d", "CPB_CampbellSoup_vol_20d", "CLX_Clorox_vol_20d", "US3Y_Rate_ret_5d", "EWA_Australia_zscore_60d"], "is_new": false}, {"model_id": "v1_h7_GLOBAL_GradientBoosting_N7", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 7, "n_features": 7, "F1_dir": 0.578, "F1_UP_FORT": 0.3371, "F1_DOWN_FORT": 0.3493, "train_start": "2001-02-07", "sampler": "SMOTE", "best_params": "{}", "features": ["NFCI_ret_5d__div__VRP", "NFCI_ret_5d__ret5x__NFCI_ret_20d", "CPB_CampbellSoup_vol_20d", "CLX_Clorox_vol_20d", "US3Y_Rate_ret_5d", "EWA_Australia_zscore_60d", "XLV_Health_zscore_60d"], "is_new": false}, {"model_id": "v1_h7_GLOBAL_GradientBoosting_N8", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 7, "n_features": 8, "F1_dir": 0.575, "F1_UP_FORT": 0.3367, "F1_DOWN_FORT": 0.3619, "train_start": "2001-02-07", "sampler": "SMOTE", "best_params": "{}", "features": ["NFCI_ret_5d__div__VRP", "NFCI_ret_5d__ret5x__NFCI_ret_20d", "CPB_CampbellSoup_vol_20d", "CLX_Clorox_vol_20d", "US3Y_Rate_ret_5d", "EWA_Australia_zscore_60d", "XLV_Health_zscore_60d", "CTAS_Cintas_vol_20d"], "is_new": false}, {"model_id": "v1_h7_GLOBAL_GradientBoosting_N9", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 7, "n_features": 9, "F1_dir": 0.5878, "F1_UP_FORT": 0.3293, "F1_DOWN_FORT": 0.3512, "train_start": "2001-02-07", "sampler": "SMOTE", "best_params": "{}", "features": ["NFCI_ret_5d__div__VRP", "NFCI_ret_5d__ret5x__NFCI_ret_20d", "CPB_CampbellSoup_vol_20d", "CLX_Clorox_vol_20d", "US3Y_Rate_ret_5d", "EWA_Australia_zscore_60d", "XLV_Health_zscore_60d", "CTAS_Cintas_vol_20d", "GILD_Gilead_ret_20d"], "is_new": false}, {"model_id": "v1_h7_GLOBAL_GradientBoosting_N10", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 7, "n_features": 10, "F1_dir": 0.585, "F1_UP_FORT": 0.3476, "F1_DOWN_FORT": 0.3945, "train_start": "2001-02-07", "sampler": "SMOTE", "best_params": "{}", "features": ["NFCI_ret_5d__div__VRP", "NFCI_ret_5d__ret5x__NFCI_ret_20d", "CPB_CampbellSoup_vol_20d", "CLX_Clorox_vol_20d", "US3Y_Rate_ret_5d", "EWA_Australia_zscore_60d", "XLV_Health_zscore_60d", "CTAS_Cintas_vol_20d", "GILD_Gilead_ret_20d", "heston_xi__prod__kalman_filtered"], "is_new": false}, {"model_id": "v1_h7_GLOBAL_GradientBoosting_N11", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 7, "n_features": 11, "F1_dir": 0.5769, "F1_UP_FORT": 0.357, "F1_DOWN_FORT": 0.3945, "train_start": "2001-02-07", "sampler": "SMOTE", "best_params": "{}", "features": ["NFCI_ret_5d__div__VRP", "NFCI_ret_5d__ret5x__NFCI_ret_20d", "CPB_CampbellSoup_vol_20d", "CLX_Clorox_vol_20d", "US3Y_Rate_ret_5d", "EWA_Australia_zscore_60d", "XLV_Health_zscore_60d", "CTAS_Cintas_vol_20d", "GILD_Gilead_ret_20d", "heston_xi__prod__kalman_filtered", "EWS_Singapore_ret_5d"], "is_new": false}, {"model_id": "v1_h7_GLOBAL_GradientBoosting_N12", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 7, "n_features": 12, "F1_dir": 0.5886, "F1_UP_FORT": 0.3561, "F1_DOWN_FORT": 0.3975, "train_start": "2001-02-07", "sampler": "SMOTE", "best_params": "{}", "features": ["NFCI_ret_5d__div__VRP", "NFCI_ret_5d__ret5x__NFCI_ret_20d", "CPB_CampbellSoup_vol_20d", "CLX_Clorox_vol_20d", "US3Y_Rate_ret_5d", "EWA_Australia_zscore_60d", "XLV_Health_zscore_60d", "CTAS_Cintas_vol_20d", "GILD_Gilead_ret_20d", "heston_xi__prod__kalman_filtered", "EWS_Singapore_ret_5d", "VVIX_vol_20d__zrel__kalman_filtered"], "is_new": false}, {"model_id": "v1_h7_GLOBAL_GradientBoosting_N13", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 7, "n_features": 13, "F1_dir": 0.5841, "F1_UP_FORT": 0.3747, "F1_DOWN_FORT": 0.4314, "train_start": "2001-02-07", "sampler": "SMOTE", "best_params": "{}", "features": ["NFCI_ret_5d__div__VRP", "NFCI_ret_5d__ret5x__NFCI_ret_20d", "CPB_CampbellSoup_vol_20d", "CLX_Clorox_vol_20d", "US3Y_Rate_ret_5d", "EWA_Australia_zscore_60d", "XLV_Health_zscore_60d", "CTAS_Cintas_vol_20d", "GILD_Gilead_ret_20d", "heston_xi__prod__kalman_filtered", "EWS_Singapore_ret_5d", "VVIX_vol_20d__zrel__kalman_filtered", "Nikkei_Japan_vol_20d"], "is_new": false}, {"model_id": "v1_h7_GLOBAL_GradientBoosting_N14", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 7, "n_features": 14, "F1_dir": 0.584, "F1_UP_FORT": 0.3625, "F1_DOWN_FORT": 0.4461, "train_start": "2001-02-07", "sampler": "SMOTE", "best_params": "{}", "features": ["NFCI_ret_5d__div__VRP", "NFCI_ret_5d__ret5x__NFCI_ret_20d", "CPB_CampbellSoup_vol_20d", "CLX_Clorox_vol_20d", "US3Y_Rate_ret_5d", "EWA_Australia_zscore_60d", "XLV_Health_zscore_60d", "CTAS_Cintas_vol_20d", "GILD_Gilead_ret_20d", "heston_xi__prod__kalman_filtered", "EWS_Singapore_ret_5d", "VVIX_vol_20d__zrel__kalman_filtered", "Nikkei_Japan_vol_20d", "SBUX_ret_5d"], "is_new": false}, {"model_id": "v1_h7_GLOBAL_GradientBoosting_N15", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 7, "n_features": 15, "F1_dir": 0.5996, "F1_UP_FORT": 0.4036, "F1_DOWN_FORT": 0.479, "train_start": "2001-02-07", "sampler": "SMOTE", "best_params": "{}", "features": ["NFCI_ret_5d__div__VRP", "NFCI_ret_5d__ret5x__NFCI_ret_20d", "CPB_CampbellSoup_vol_20d", "CLX_Clorox_vol_20d", "US3Y_Rate_ret_5d", "EWA_Australia_zscore_60d", "XLV_Health_zscore_60d", "CTAS_Cintas_vol_20d", "GILD_Gilead_ret_20d", "heston_xi__prod__kalman_filtered", "EWS_Singapore_ret_5d", "VVIX_vol_20d__zrel__kalman_filtered", "Nikkei_Japan_vol_20d", "SBUX_ret_5d", "VVIX_vol_20d__prod__kalman_filtered"], "is_new": false}, {"model_id": "v1_h7_GLOBAL_GradientBoosting_N16", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 7, "n_features": 16, "F1_dir": 0.6156, "F1_UP_FORT": 0.4146, "F1_DOWN_FORT": 0.4755, "train_start": "2001-02-07", "sampler": "SMOTE", "best_params": "{}", "features": ["NFCI_ret_5d__div__VRP", "NFCI_ret_5d__ret5x__NFCI_ret_20d", "CPB_CampbellSoup_vol_20d", "CLX_Clorox_vol_20d", "US3Y_Rate_ret_5d", "EWA_Australia_zscore_60d", "XLV_Health_zscore_60d", "CTAS_Cintas_vol_20d", "GILD_Gilead_ret_20d", "heston_xi__prod__kalman_filtered", "EWS_Singapore_ret_5d", "VVIX_vol_20d__zrel__kalman_filtered", "Nikkei_Japan_vol_20d", "SBUX_ret_5d", "VVIX_vol_20d__prod__kalman_filtered", "PAYX_Paychex_ret_20d"], "is_new": false}, {"model_id": "v1_h7_GLOBAL_GradientBoosting_N17", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 7, "n_features": 17, "F1_dir": 0.5963, "F1_UP_FORT": 0.404, "F1_DOWN_FORT": 0.4853, "train_start": "2001-02-07", "sampler": "SMOTE", "best_params": "{}", "features": ["NFCI_ret_5d__div__VRP", "NFCI_ret_5d__ret5x__NFCI_ret_20d", "CPB_CampbellSoup_vol_20d", "CLX_Clorox_vol_20d", "US3Y_Rate_ret_5d", "EWA_Australia_zscore_60d", "XLV_Health_zscore_60d", "CTAS_Cintas_vol_20d", "GILD_Gilead_ret_20d", "heston_xi__prod__kalman_filtered", "EWS_Singapore_ret_5d", "VVIX_vol_20d__zrel__kalman_filtered", "Nikkei_Japan_vol_20d", "SBUX_ret_5d", "VVIX_vol_20d__prod__kalman_filtered", "PAYX_Paychex_ret_20d", "NFCI_ret_5d__div__NFCI_vol_20d"], "is_new": false}, {"model_id": "v1_h7_GLOBAL_RandomForest_N5", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 7, "n_features": 5, "F1_dir": 0.5267, "F1_UP_FORT": 0.3324, "F1_DOWN_FORT": 0.3011, "train_start": "2001-02-07", "sampler": "SMOTE", "best_params": "{}", "features": ["NFCI_ret_5d__div__VRP", "NFCI_ret_5d__ret5x__NFCI_ret_20d", "CPB_CampbellSoup_vol_20d", "CLX_Clorox_vol_20d", "US3Y_Rate_ret_5d"], "is_new": false}, {"model_id": "v1_h7_GLOBAL_RandomForest_N6", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 7, "n_features": 6, "F1_dir": 0.5656, "F1_UP_FORT": 0.3276, "F1_DOWN_FORT": 0.3579, "train_start": "2001-02-07", "sampler": "SMOTE", "best_params": "{}", "features": ["NFCI_ret_5d__div__VRP", "NFCI_ret_5d__ret5x__NFCI_ret_20d", "CPB_CampbellSoup_vol_20d", "CLX_Clorox_vol_20d", "US3Y_Rate_ret_5d", "EWA_Australia_zscore_60d"], "is_new": false}, {"model_id": "v1_h7_GLOBAL_RandomForest_N7", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 7, "n_features": 7, "F1_dir": 0.5687, "F1_UP_FORT": 0.3419, "F1_DOWN_FORT": 0.3873, "train_start": "2001-02-07", "sampler": "SMOTE", "best_params": "{}", "features": ["NFCI_ret_5d__div__VRP", "NFCI_ret_5d__ret5x__NFCI_ret_20d", "CPB_CampbellSoup_vol_20d", "CLX_Clorox_vol_20d", "US3Y_Rate_ret_5d", "EWA_Australia_zscore_60d", "XLV_Health_zscore_60d"], "is_new": false}, {"model_id": "v1_h7_GLOBAL_RandomForest_N8", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 7, "n_features": 8, "F1_dir": 0.5713, "F1_UP_FORT": 0.3245, "F1_DOWN_FORT": 0.3995, "train_start": "2001-02-07", "sampler": "SMOTE", "best_params": "{}", "features": ["NFCI_ret_5d__div__VRP", "NFCI_ret_5d__ret5x__NFCI_ret_20d", "CPB_CampbellSoup_vol_20d", "CLX_Clorox_vol_20d", "US3Y_Rate_ret_5d", "EWA_Australia_zscore_60d", "XLV_Health_zscore_60d", "CTAS_Cintas_vol_20d"], "is_new": false}, {"model_id": "v1_h7_GLOBAL_RandomForest_N9", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 7, "n_features": 9, "F1_dir": 0.5809, "F1_UP_FORT": 0.3249, "F1_DOWN_FORT": 0.4081, "train_start": "2001-02-07", "sampler": "SMOTE", "best_params": "{}", "features": ["NFCI_ret_5d__div__VRP", "NFCI_ret_5d__ret5x__NFCI_ret_20d", "CPB_CampbellSoup_vol_20d", "CLX_Clorox_vol_20d", "US3Y_Rate_ret_5d", "EWA_Australia_zscore_60d", "XLV_Health_zscore_60d", "CTAS_Cintas_vol_20d", "GILD_Gilead_ret_20d"], "is_new": false}, {"model_id": "v1_h7_GLOBAL_RandomForest_N10", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 7, "n_features": 10, "F1_dir": 0.5552, "F1_UP_FORT": 0.3612, "F1_DOWN_FORT": 0.3978, "train_start": "2001-02-07", "sampler": "SMOTE", "best_params": "{}", "features": ["NFCI_ret_5d__div__VRP", "NFCI_ret_5d__ret5x__NFCI_ret_20d", "CPB_CampbellSoup_vol_20d", "CLX_Clorox_vol_20d", "US3Y_Rate_ret_5d", "EWA_Australia_zscore_60d", "XLV_Health_zscore_60d", "CTAS_Cintas_vol_20d", "GILD_Gilead_ret_20d", "heston_xi__prod__kalman_filtered"], "is_new": false}, {"model_id": "v1_h7_GLOBAL_RandomForest_N11", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 7, "n_features": 11, "F1_dir": 0.5498, "F1_UP_FORT": 0.3685, "F1_DOWN_FORT": 0.4142, "train_start": "2001-02-07", "sampler": "SMOTE", "best_params": "{}", "features": ["NFCI_ret_5d__div__VRP", "NFCI_ret_5d__ret5x__NFCI_ret_20d", "CPB_CampbellSoup_vol_20d", "CLX_Clorox_vol_20d", "US3Y_Rate_ret_5d", "EWA_Australia_zscore_60d", "XLV_Health_zscore_60d", "CTAS_Cintas_vol_20d", "GILD_Gilead_ret_20d", "heston_xi__prod__kalman_filtered", "EWS_Singapore_ret_5d"], "is_new": false}, {"model_id": "v1_h7_GLOBAL_RandomForest_N12", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 7, "n_features": 12, "F1_dir": 0.5664, "F1_UP_FORT": 0.3824, "F1_DOWN_FORT": 0.4237, "train_start": "2001-02-07", "sampler": "SMOTE", "best_params": "{}", "features": ["NFCI_ret_5d__div__VRP", "NFCI_ret_5d__ret5x__NFCI_ret_20d", "CPB_CampbellSoup_vol_20d", "CLX_Clorox_vol_20d", "US3Y_Rate_ret_5d", "EWA_Australia_zscore_60d", "XLV_Health_zscore_60d", "CTAS_Cintas_vol_20d", "GILD_Gilead_ret_20d", "heston_xi__prod__kalman_filtered", "EWS_Singapore_ret_5d", "VVIX_vol_20d__zrel__kalman_filtered"], "is_new": false}, {"model_id": "v1_h7_GLOBAL_RandomForest_N13", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 7, "n_features": 13, "F1_dir": 0.5518, "F1_UP_FORT": 0.3493, "F1_DOWN_FORT": 0.41, "train_start": "2001-02-07", "sampler": "SMOTE", "best_params": "{}", "features": ["NFCI_ret_5d__div__VRP", "NFCI_ret_5d__ret5x__NFCI_ret_20d", "CPB_CampbellSoup_vol_20d", "CLX_Clorox_vol_20d", "US3Y_Rate_ret_5d", "EWA_Australia_zscore_60d", "XLV_Health_zscore_60d", "CTAS_Cintas_vol_20d", "GILD_Gilead_ret_20d", "heston_xi__prod__kalman_filtered", "EWS_Singapore_ret_5d", "VVIX_vol_20d__zrel__kalman_filtered", "Nikkei_Japan_vol_20d"], "is_new": false}, {"model_id": "v1_h7_GLOBAL_RandomForest_N14", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 7, "n_features": 14, "F1_dir": 0.5491, "F1_UP_FORT": 0.3277, "F1_DOWN_FORT": 0.4215, "train_start": "2001-02-07", "sampler": "SMOTE", "best_params": "{}", "features": ["NFCI_ret_5d__div__VRP", "NFCI_ret_5d__ret5x__NFCI_ret_20d", "CPB_CampbellSoup_vol_20d", "CLX_Clorox_vol_20d", "US3Y_Rate_ret_5d", "EWA_Australia_zscore_60d", "XLV_Health_zscore_60d", "CTAS_Cintas_vol_20d", "GILD_Gilead_ret_20d", "heston_xi__prod__kalman_filtered", "EWS_Singapore_ret_5d", "VVIX_vol_20d__zrel__kalman_filtered", "Nikkei_Japan_vol_20d", "SBUX_ret_5d"], "is_new": false}, {"model_id": "v1_h7_GLOBAL_RandomForest_N15", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 7, "n_features": 15, "F1_dir": 0.5791, "F1_UP_FORT": 0.3934, "F1_DOWN_FORT": 0.4547, "train_start": "2001-02-07", "sampler": "SMOTE", "best_params": "{}", "features": ["NFCI_ret_5d__div__VRP", "NFCI_ret_5d__ret5x__NFCI_ret_20d", "CPB_CampbellSoup_vol_20d", "CLX_Clorox_vol_20d", "US3Y_Rate_ret_5d", "EWA_Australia_zscore_60d", "XLV_Health_zscore_60d", "CTAS_Cintas_vol_20d", "GILD_Gilead_ret_20d", "heston_xi__prod__kalman_filtered", "EWS_Singapore_ret_5d", "VVIX_vol_20d__zrel__kalman_filtered", "Nikkei_Japan_vol_20d", "SBUX_ret_5d", "VVIX_vol_20d__prod__kalman_filtered"], "is_new": false}, {"model_id": "v1_h7_GLOBAL_RandomForest_N16", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 7, "n_features": 16, "F1_dir": 0.5742, "F1_UP_FORT": 0.3703, "F1_DOWN_FORT": 0.4522, "train_start": "2001-02-07", "sampler": "SMOTE", "best_params": "{}", "features": ["NFCI_ret_5d__div__VRP", "NFCI_ret_5d__ret5x__NFCI_ret_20d", "CPB_CampbellSoup_vol_20d", "CLX_Clorox_vol_20d", "US3Y_Rate_ret_5d", "EWA_Australia_zscore_60d", "XLV_Health_zscore_60d", "CTAS_Cintas_vol_20d", "GILD_Gilead_ret_20d", "heston_xi__prod__kalman_filtered", "EWS_Singapore_ret_5d", "VVIX_vol_20d__zrel__kalman_filtered", "Nikkei_Japan_vol_20d", "SBUX_ret_5d", "VVIX_vol_20d__prod__kalman_filtered", "PAYX_Paychex_ret_20d"], "is_new": false}, {"model_id": "v1_h7_GLOBAL_RandomForest_N17", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 7, "n_features": 17, "F1_dir": 0.6014, "F1_UP_FORT": 0.412, "F1_DOWN_FORT": 0.4751, "train_start": "2001-02-07", "sampler": "SMOTE", "best_params": "{}", "features": ["NFCI_ret_5d__div__VRP", "NFCI_ret_5d__ret5x__NFCI_ret_20d", "CPB_CampbellSoup_vol_20d", "CLX_Clorox_vol_20d", "US3Y_Rate_ret_5d", "EWA_Australia_zscore_60d", "XLV_Health_zscore_60d", "CTAS_Cintas_vol_20d", "GILD_Gilead_ret_20d", "heston_xi__prod__kalman_filtered", "EWS_Singapore_ret_5d", "VVIX_vol_20d__zrel__kalman_filtered", "Nikkei_Japan_vol_20d", "SBUX_ret_5d", "VVIX_vol_20d__prod__kalman_filtered", "PAYX_Paychex_ret_20d", "NFCI_ret_5d__div__NFCI_vol_20d"], "is_new": false}, {"model_id": "v1_h7_GLOBAL_LogisticRegression_N6", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 7, "n_features": 6, "F1_dir": 0.5479, "F1_UP_FORT": 0.3027, "F1_DOWN_FORT": 0.4343, "train_start": "2001-02-07", "sampler": "SMOTE", "best_params": "{}", "features": ["NFCI_ret_5d__div__VRP", "NFCI_ret_5d__ret5x__NFCI_ret_20d", "CPB_CampbellSoup_vol_20d", "CLX_Clorox_vol_20d", "US3Y_Rate_ret_5d", "EWA_Australia_zscore_60d"], "is_new": false}, {"model_id": "v1_h7_GLOBAL_LogisticRegression_N7", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 7, "n_features": 7, "F1_dir": 0.5666, "F1_UP_FORT": 0.2675, "F1_DOWN_FORT": 0.4093, "train_start": "2001-02-07", "sampler": "SMOTE", "best_params": "{}", "features": ["NFCI_ret_5d__div__VRP", "NFCI_ret_5d__ret5x__NFCI_ret_20d", "CPB_CampbellSoup_vol_20d", "CLX_Clorox_vol_20d", "US3Y_Rate_ret_5d", "EWA_Australia_zscore_60d", "XLV_Health_zscore_60d"], "is_new": false}, {"model_id": "v1_h7_GLOBAL_LogisticRegression_N8", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 7, "n_features": 8, "F1_dir": 0.5981, "F1_UP_FORT": 0.257, "F1_DOWN_FORT": 0.4324, "train_start": "2001-02-07", "sampler": "SMOTE", "best_params": "{}", "features": ["NFCI_ret_5d__div__VRP", "NFCI_ret_5d__ret5x__NFCI_ret_20d", "CPB_CampbellSoup_vol_20d", "CLX_Clorox_vol_20d", "US3Y_Rate_ret_5d", "EWA_Australia_zscore_60d", "XLV_Health_zscore_60d", "CTAS_Cintas_vol_20d"], "is_new": false}, {"model_id": "v1_h7_GLOBAL_LogisticRegression_N9", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 7, "n_features": 9, "F1_dir": 0.5928, "F1_UP_FORT": 0.2463, "F1_DOWN_FORT": 0.4281, "train_start": "2001-02-07", "sampler": "SMOTE", "best_params": "{}", "features": ["NFCI_ret_5d__div__VRP", "NFCI_ret_5d__ret5x__NFCI_ret_20d", "CPB_CampbellSoup_vol_20d", "CLX_Clorox_vol_20d", "US3Y_Rate_ret_5d", "EWA_Australia_zscore_60d", "XLV_Health_zscore_60d", "CTAS_Cintas_vol_20d", "GILD_Gilead_ret_20d"], "is_new": false}, {"model_id": "v1_h7_GLOBAL_LogisticRegression_N10", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 7, "n_features": 10, "F1_dir": 0.6229, "F1_UP_FORT": 0.3255, "F1_DOWN_FORT": 0.4538, "train_start": "2001-02-07", "sampler": "SMOTE", "best_params": "{}", "features": ["NFCI_ret_5d__div__VRP", "NFCI_ret_5d__ret5x__NFCI_ret_20d", "CPB_CampbellSoup_vol_20d", "CLX_Clorox_vol_20d", "US3Y_Rate_ret_5d", "EWA_Australia_zscore_60d", "XLV_Health_zscore_60d", "CTAS_Cintas_vol_20d", "GILD_Gilead_ret_20d", "heston_xi__prod__kalman_filtered"], "is_new": false}, {"model_id": "v1_h7_GLOBAL_LogisticRegression_N11", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 7, "n_features": 11, "F1_dir": 0.6208, "F1_UP_FORT": 0.3312, "F1_DOWN_FORT": 0.4469, "train_start": "2001-02-07", "sampler": "SMOTE", "best_params": "{}", "features": ["NFCI_ret_5d__div__VRP", "NFCI_ret_5d__ret5x__NFCI_ret_20d", "CPB_CampbellSoup_vol_20d", "CLX_Clorox_vol_20d", "US3Y_Rate_ret_5d", "EWA_Australia_zscore_60d", "XLV_Health_zscore_60d", "CTAS_Cintas_vol_20d", "GILD_Gilead_ret_20d", "heston_xi__prod__kalman_filtered", "EWS_Singapore_ret_5d"], "is_new": false}, {"model_id": "v1_h7_GLOBAL_LogisticRegression_N12", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 7, "n_features": 12, "F1_dir": 0.604, "F1_UP_FORT": 0.3539, "F1_DOWN_FORT": 0.4483, "train_start": "2001-02-07", "sampler": "SMOTE", "best_params": "{}", "features": ["NFCI_ret_5d__div__VRP", "NFCI_ret_5d__ret5x__NFCI_ret_20d", "CPB_CampbellSoup_vol_20d", "CLX_Clorox_vol_20d", "US3Y_Rate_ret_5d", "EWA_Australia_zscore_60d", "XLV_Health_zscore_60d", "CTAS_Cintas_vol_20d", "GILD_Gilead_ret_20d", "heston_xi__prod__kalman_filtered", "EWS_Singapore_ret_5d", "VVIX_vol_20d__zrel__kalman_filtered"], "is_new": false}, {"model_id": "v1_h7_GLOBAL_LogisticRegression_N13", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 7, "n_features": 13, "F1_dir": 0.5946, "F1_UP_FORT": 0.3577, "F1_DOWN_FORT": 0.4426, "train_start": "2001-02-07", "sampler": "SMOTE", "best_params": "{}", "features": ["NFCI_ret_5d__div__VRP", "NFCI_ret_5d__ret5x__NFCI_ret_20d", "CPB_CampbellSoup_vol_20d", "CLX_Clorox_vol_20d", "US3Y_Rate_ret_5d", "EWA_Australia_zscore_60d", "XLV_Health_zscore_60d", "CTAS_Cintas_vol_20d", "GILD_Gilead_ret_20d", "heston_xi__prod__kalman_filtered", "EWS_Singapore_ret_5d", "VVIX_vol_20d__zrel__kalman_filtered", "Nikkei_Japan_vol_20d"], "is_new": false}, {"model_id": "v1_h7_GLOBAL_LogisticRegression_N14", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 7, "n_features": 14, "F1_dir": 0.5977, "F1_UP_FORT": 0.365, "F1_DOWN_FORT": 0.45, "train_start": "2001-02-07", "sampler": "SMOTE", "best_params": "{}", "features": ["NFCI_ret_5d__div__VRP", "NFCI_ret_5d__ret5x__NFCI_ret_20d", "CPB_CampbellSoup_vol_20d", "CLX_Clorox_vol_20d", "US3Y_Rate_ret_5d", "EWA_Australia_zscore_60d", "XLV_Health_zscore_60d", "CTAS_Cintas_vol_20d", "GILD_Gilead_ret_20d", "heston_xi__prod__kalman_filtered", "EWS_Singapore_ret_5d", "VVIX_vol_20d__zrel__kalman_filtered", "Nikkei_Japan_vol_20d", "SBUX_ret_5d"], "is_new": false}, {"model_id": "v1_h7_GLOBAL_LogisticRegression_N15", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 7, "n_features": 15, "F1_dir": 0.6053, "F1_UP_FORT": 0.3615, "F1_DOWN_FORT": 0.4751, "train_start": "2001-02-07", "sampler": "SMOTE", "best_params": "{}", "features": ["NFCI_ret_5d__div__VRP", "NFCI_ret_5d__ret5x__NFCI_ret_20d", "CPB_CampbellSoup_vol_20d", "CLX_Clorox_vol_20d", "US3Y_Rate_ret_5d", "EWA_Australia_zscore_60d", "XLV_Health_zscore_60d", "CTAS_Cintas_vol_20d", "GILD_Gilead_ret_20d", "heston_xi__prod__kalman_filtered", "EWS_Singapore_ret_5d", "VVIX_vol_20d__zrel__kalman_filtered", "Nikkei_Japan_vol_20d", "SBUX_ret_5d", "VVIX_vol_20d__prod__kalman_filtered"], "is_new": false}, {"model_id": "v1_h7_GLOBAL_LogisticRegression_N16", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 7, "n_features": 16, "F1_dir": 0.578, "F1_UP_FORT": 0.3814, "F1_DOWN_FORT": 0.4581, "train_start": "2001-02-07", "sampler": "SMOTE", "best_params": "{}", "features": ["NFCI_ret_5d__div__VRP", "NFCI_ret_5d__ret5x__NFCI_ret_20d", "CPB_CampbellSoup_vol_20d", "CLX_Clorox_vol_20d", "US3Y_Rate_ret_5d", "EWA_Australia_zscore_60d", "XLV_Health_zscore_60d", "CTAS_Cintas_vol_20d", "GILD_Gilead_ret_20d", "heston_xi__prod__kalman_filtered", "EWS_Singapore_ret_5d", "VVIX_vol_20d__zrel__kalman_filtered", "Nikkei_Japan_vol_20d", "SBUX_ret_5d", "VVIX_vol_20d__prod__kalman_filtered", "PAYX_Paychex_ret_20d"], "is_new": false}, {"model_id": "v1_h7_GLOBAL_LogisticRegression_N17", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 7, "n_features": 17, "F1_dir": 0.6144, "F1_UP_FORT": 0.4294, "F1_DOWN_FORT": 0.4817, "train_start": "2001-02-07", "sampler": "SMOTE", "best_params": "{}", "features": ["NFCI_ret_5d__div__VRP", "NFCI_ret_5d__ret5x__NFCI_ret_20d", "CPB_CampbellSoup_vol_20d", "CLX_Clorox_vol_20d", "US3Y_Rate_ret_5d", "EWA_Australia_zscore_60d", "XLV_Health_zscore_60d", "CTAS_Cintas_vol_20d", "GILD_Gilead_ret_20d", "heston_xi__prod__kalman_filtered", "EWS_Singapore_ret_5d", "VVIX_vol_20d__zrel__kalman_filtered", "Nikkei_Japan_vol_20d", "SBUX_ret_5d", "VVIX_vol_20d__prod__kalman_filtered", "PAYX_Paychex_ret_20d", "NFCI_ret_5d__div__NFCI_vol_20d"], "is_new": false}, {"model_id": "v1_h7_GLOBAL_RandomForest_Optuna_N17", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 7, "n_features": 17, "F1_dir": 0.5887, "F1_UP_FORT": 0.419, "F1_DOWN_FORT": 0.4789, "train_start": "2001-02-07", "sampler": "SMOTE", "best_params": "{}", "features": ["NFCI_ret_5d__div__VRP", "NFCI_ret_5d__ret5x__NFCI_ret_20d", "CPB_CampbellSoup_vol_20d", "CLX_Clorox_vol_20d", "US3Y_Rate_ret_5d", "EWA_Australia_zscore_60d", "XLV_Health_zscore_60d", "CTAS_Cintas_vol_20d", "GILD_Gilead_ret_20d", "heston_xi__prod__kalman_filtered", "EWS_Singapore_ret_5d", "VVIX_vol_20d__zrel__kalman_filtered", "Nikkei_Japan_vol_20d", "SBUX_ret_5d", "VVIX_vol_20d__prod__kalman_filtered", "PAYX_Paychex_ret_20d", "NFCI_ret_5d__div__NFCI_vol_20d"], "is_new": false}, {"model_id": "v1_h7_GLOBAL_RandomForest_OptunaCal_N17", "algo": "RandomForestCal", "regime": "GLOBAL", "horizon": 7, "n_features": 17, "F1_dir": 0.5864, "F1_UP_FORT": 0.4079, "F1_DOWN_FORT": 0.4895, "train_start": "2001-02-07", "sampler": "SMOTE", "best_params": "{}", "features": ["NFCI_ret_5d__div__VRP", "NFCI_ret_5d__ret5x__NFCI_ret_20d", "CPB_CampbellSoup_vol_20d", "CLX_Clorox_vol_20d", "US3Y_Rate_ret_5d", "EWA_Australia_zscore_60d", "XLV_Health_zscore_60d", "CTAS_Cintas_vol_20d", "GILD_Gilead_ret_20d", "heston_xi__prod__kalman_filtered", "EWS_Singapore_ret_5d", "VVIX_vol_20d__zrel__kalman_filtered", "Nikkei_Japan_vol_20d", "SBUX_ret_5d", "VVIX_vol_20d__prod__kalman_filtered", "PAYX_Paychex_ret_20d", "NFCI_ret_5d__div__NFCI_vol_20d"], "is_new": false}, {"model_id": "new_h1_CALM_XGBoost_N5_t0", "algo": "XGBoost", "regime": "CALM", "horizon": 1, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWS_Singapore_ret_5d", "EWM_Malaysia_ret_1d", "DHR_ret_1d"], "is_new": true}, {"model_id": "new_h1_CALM_XGBoost_N5_t1", "algo": "XGBoost", "regime": "CALM", "horizon": 1, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "ASX_Australia_ret_5d", "AORD_AUS_zscore_60d", "CMCSA_ret_1d"], "is_new": true}, {"model_id": "new_h1_CALM_XGBoost_N5_t2", "algo": "XGBoost", "regime": "CALM", "horizon": 1, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "hmm_p_stress", "TXN_vol_20d", "NVDA_vol_20d"], "is_new": true}, {"model_id": "new_h1_CALM_XGBoost_N5_t3", "algo": "XGBoost", "regime": "CALM", "horizon": 1, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EFFR_vol_20d", "NWL_Newell_ret_20d", "JNJ_ret_1d"], "is_new": true}, {"model_id": "new_h1_CALM_XGBoost_N5_t4", "algo": "XGBoost", "regime": "CALM", "horizon": 1, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "CPB_CampbellSoup_vol_20d", "CMCSA_ret_1d", "MSTR_Bitcoin3_ret_20d"], "is_new": true}, {"model_id": "new_h1_CALM_XGBoost_N5_t5", "algo": "XGBoost", "regime": "CALM", "horizon": 1, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "AXP_Amex_ret_20d", "PFE_ret_1d", "TM_Telephone_ret_1d"], "is_new": true}, {"model_id": "new_h1_CALM_XGBoost_N5_t6", "algo": "XGBoost", "regime": "CALM", "horizon": 1, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "XOM_ret_20d", "XLY_Disc_vol_20d", "LOW_Lowes_ret_5d"], "is_new": true}, {"model_id": "new_h1_CALM_XGBoost_N5_t7", "algo": "XGBoost", "regime": "CALM", "horizon": 1, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "spx_vol_5d", "EWM_Malaysia_vol_20d", "HD_zscore_60d"], "is_new": true}, {"model_id": "new_h1_CALM_XGBoost_N8_t0", "algo": "XGBoost", "regime": "CALM", "horizon": 1, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "HD_ret_1d", "PCAR_PaccarInc_ret_5d", "heston_ev_h3", "LUV_SouthwestAir_ret_5d", "EMR_Emerson_ret_20d", "MO_AltriaMG_ret_1d"], "is_new": true}, {"model_id": "new_h1_CALM_XGBoost_N8_t1", "algo": "XGBoost", "regime": "CALM", "horizon": 1, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "XLF_Fin_vol_20d", "heston_ev_h3", "EOG_EOGResources_ret_5d", "TGT_Target_zscore_60d", "EWS_Singapore_ret_5d", "EMR_Emerson_ret_20d"], "is_new": true}, {"model_id": "new_h1_CALM_XGBoost_N8_t2", "algo": "XGBoost", "regime": "CALM", "horizon": 1, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "IWM_SmallCap_vol_20d", "BDX_Becton_Dickinson_ret_20d", "EWM_Malaysia_vol_20d", "ASX_Australia_ret_5d", "INTC_ret_1d", "M_Macys_vol_20d"], "is_new": true}, {"model_id": "new_h1_CALM_XGBoost_N8_t3", "algo": "XGBoost", "regime": "CALM", "horizon": 1, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWJ_Japan_vol_20d", "spx_abs_ret_max_5d", "SJM_JM_Smucker_ret_1d", "MS_MorganStanley_zscore_60d", "heston_var_ev_h5", "PFE_ret_1d"], "is_new": true}, {"model_id": "new_h1_CALM_XGBoost_N8_t4", "algo": "XGBoost", "regime": "CALM", "horizon": 1, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "US30Y_Rate_ret_20d", "TM_Telephone_ret_1d", "FedFunds_zscore_60d", "US7Y_Rate_ret_20d", "Nikkei_Japan_vol_20d", "M_Macys_vol_20d"], "is_new": true}, {"model_id": "new_h1_CALM_XGBoost_N8_t5", "algo": "XGBoost", "regime": "CALM", "horizon": 1, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "CMCSA_ret_1d", "AXP_Amex_vol_20d", "CTAS_Cintas_vol_20d", "EFFR_ret_1d", "XOM_ret_20d", "EFFR_vol_20d"], "is_new": true}, {"model_id": "new_h1_CALM_XGBoost_N8_t6", "algo": "XGBoost", "regime": "CALM", "horizon": 1, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "MSTR_Bitcoin3_ret_20d", "VVIX_ret_20d", "PLD_Prologis_ret_5d", "TED_Spread_zscore_60d", "EWY_Korea_zscore_60d", "US1Y_Rate_ret_20d"], "is_new": true}, {"model_id": "new_h1_CALM_XGBoost_N8_t7", "algo": "XGBoost", "regime": "CALM", "horizon": 1, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "CTAS_Cintas_vol_20d", "HUM_Humana_ret_5d", "ORCL_vol_20d", "EFFR_vol_20d", "PCAR_PaccarInc_ret_5d", "3M_ret_5d"], "is_new": true}, {"model_id": "new_h1_CALM_XGBoost_N10_t0", "algo": "XGBoost", "regime": "CALM", "horizon": 1, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "XOM_ret_20d", "HangSeng_HK_ret_1d", "heston_var_ev_h3", "SLB_Schlumberger_ret_1d", "XLF_Fin_vol_20d", "EWC_Canada_zscore_60d", "EFFR_ret_1d", "MRK_Merck_zscore_60d"], "is_new": true}, {"model_id": "new_h1_CALM_XGBoost_N10_t1", "algo": "XGBoost", "regime": "CALM", "horizon": 1, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "AMGN_Amgen_ret_1d", "CTAS_Cintas_vol_20d", "LOW_Lowes_ret_5d", "HangSeng_HK_vol_20d", "EWQ_France_zscore_60d", "BDX_Becton_Dickinson_ret_20d", "US3M_Rate_zscore_60d", "CCI_CrownCastle_vol_20d"], "is_new": true}, {"model_id": "new_h1_CALM_XGBoost_N10_t2", "algo": "XGBoost", "regime": "CALM", "horizon": 1, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "DAX_Germany_zscore_60d", "spx_momentum_3d", "QQQ_vol_20d", "T_ret_1d", "ENB_EnbridgeInc_ret_1d", "FedFunds_zscore_60d", "TED_Spread_zscore_60d", "XLF_Fin_vol_20d"], "is_new": true}, {"model_id": "new_h1_CALM_XGBoost_N10_t3", "algo": "XGBoost", "regime": "CALM", "horizon": 1, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "M_Macys_vol_20d", "vix_mean_abs_ret_5d", "hmm_p_stress", "MRK_Merck_zscore_60d", "ORCL_zscore_60d", "SPY_zscore_60d", "PAYX_Paychex_zscore_60d", "Core_PCE_zscore_60d"], "is_new": true}, {"model_id": "new_h1_CALM_XGBoost_N10_t4", "algo": "XGBoost", "regime": "CALM", "horizon": 1, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "AMT_AmericanTower_ret_1d", "VOD_Vodafone_zscore_60d", "US1Y_Rate_ret_20d", "EWM_Malaysia_vol_20d", "EWL_Switzerland_zscore_60d", "3M_vol_20d", "Brent_Oil_FRED_ret_20d", "gjr_condvar_h1"], "is_new": true}, {"model_id": "new_h1_CALM_XGBoost_N10_t5", "algo": "XGBoost", "regime": "CALM", "horizon": 1, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "IYM_BasicMaterials_ret_20d", "Brent_Oil_FRED_ret_20d", "SO_SouthernCo_ret_5d", "EWQ_France_ret_20d", "HD_ret_1d", "vix_mean_abs_ret_5d", "SBUX_vol_20d", "EWJ_Japan_vol_20d"], "is_new": true}, {"model_id": "new_h1_CALM_XGBoost_N10_t6", "algo": "XGBoost", "regime": "CALM", "horizon": 1, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "QQQ_vol_20d", "NEE_NextEra_ret_20d", "EWA_Australia_ret_1d", "CPB_CampbellSoup_ret_20d", "spx_momentum_3d", "Nikkei_Japan_zscore_60d", "LMT_LockheedMartin_vol_20d", "NWL_Newell_ret_20d"], "is_new": true}, {"model_id": "new_h1_CALM_XGBoost_N10_t7", "algo": "XGBoost", "regime": "CALM", "horizon": 1, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "BLK_BlackRock_zscore_60d", "Michigan_Sentiment_ret_20d", "Industrial_Production_zscore_60d", "CPB_CampbellSoup_zscore_60d", "EWY_Korea_ret_20d", "EWA_Australia_zscore_60d", "EWA_Australia_ret_1d", "TM_Telephone_ret_1d"], "is_new": true}, {"model_id": "new_h1_CALM_XGBoost_N12_t0", "algo": "XGBoost", "regime": "CALM", "horizon": 1, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "HD_ret_1d", "PG_ret_20d", "EWY_Korea_ret_20d", "Brent_Oil_FRED_ret_5d", "PAYX_Paychex_ret_20d", "T_ret_1d", "CPB_CampbellSoup_ret_5d", "AXP_Amex_vol_20d", "hmm_p_stress", "TM_Telephone_vol_20d"], "is_new": true}, {"model_id": "new_h1_CALM_XGBoost_N12_t1", "algo": "XGBoost", "regime": "CALM", "horizon": 1, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "NEE_NextEra_ret_20d", "SBUX_vol_20d", "SBUX_ret_5d", "EWJ_Japan_vol_20d", "M_Macys_vol_20d", "BLK_BlackRock_zscore_60d", "ENB_EnbridgeInc_ret_1d", "MSTR_Bitcoin3_ret_20d", "3M_ret_5d", "MS_MorganStanley_ret_1d"], "is_new": true}, {"model_id": "new_h1_CALM_XGBoost_N12_t2", "algo": "XGBoost", "regime": "CALM", "horizon": 1, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "US1Y_Rate_ret_5d", "spx_vol_5d", "SBUX_vol_20d", "EFFR_ret_1d", "EWA_Australia_ret_1d", "HUM_Humana_ret_5d", "EWL_Switzerland_vol_20d", "BDX_Becton_Dickinson_ret_20d", "US6M_Rate_ret_20d", "TM_Telephone_vol_20d"], "is_new": true}, {"model_id": "new_h1_CALM_XGBoost_N12_t3", "algo": "XGBoost", "regime": "CALM", "horizon": 1, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "US7Y_Rate_ret_20d", "Retail_Sales_zscore_60d", "SJM_JM_Smucker_ret_5d", "TED_Spread_zscore_60d", "EFFR_vol_20d", "BDX_Becton_Dickinson_ret_20d", "SLB_Schlumberger_ret_5d", "CCI_CrownCastle_vol_20d", "ES_Evergy_ret_1d", "BTI_BritishAmerican_ret_5d"], "is_new": true}, {"model_id": "new_h1_CALM_XGBoost_N12_t4", "algo": "XGBoost", "regime": "CALM", "horizon": 1, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "Core_PCE_zscore_60d", "US3M_Rate_zscore_60d", "EWY_Korea_ret_20d", "US6M_Rate_ret_20d", "VVIX_ret_20d", "AVB_AvalonBay_zscore_60d", "XLY_Disc_vol_20d", "CCI_CrownCastle_vol_20d", "NVDA_vol_20d", "spx_momentum_3d"], "is_new": true}, {"model_id": "new_h1_CALM_XGBoost_N12_t5", "algo": "XGBoost", "regime": "CALM", "horizon": 1, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "vix_mean_abs_ret_5d", "INTC_ret_1d", "EWQ_France_zscore_60d", "FedFunds_zscore_60d", "MS_MorganStanley_zscore_60d", "DE_Deere_ret_5d", "heston_var_ev_h7", "HangSeng_HK_ret_1d", "PFE_ret_1d", "Brent_Oil_FRED_ret_20d"], "is_new": true}, {"model_id": "new_h1_CALM_XGBoost_N12_t6", "algo": "XGBoost", "regime": "CALM", "horizon": 1, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "TM_Telephone_ret_1d", "EWJ_Japan_vol_20d", "SLB_Schlumberger_ret_1d", "FedFunds_zscore_60d", "DE_Deere_ret_5d", "LMT_LockheedMartin_ret_1d", "BTI_BritishAmerican_ret_5d", "EWS_Singapore_ret_5d", "MO_AltriaMG_ret_1d", "HD_ret_20d"], "is_new": true}, {"model_id": "new_h1_CALM_XGBoost_N12_t7", "algo": "XGBoost", "regime": "CALM", "horizon": 1, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "vix_acceleration_1d", "US1Y_Rate_ret_20d", "HangSeng_HK_ret_1d", "spx_vol_5d", "CPB_CampbellSoup_vol_20d", "DE_Deere_vol_20d", "FedFunds_zscore_60d", "Core_CPI_zscore_60d", "CPB_CampbellSoup_zscore_60d", "EFFR_ret_1d"], "is_new": true}, {"model_id": "new_h1_CALM_XGBoost_N15_t0", "algo": "XGBoost", "regime": "CALM", "horizon": 1, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWG_Germany_vol_20d", "heston_ev_h3", "FedFunds_zscore_60d", "SLB_Schlumberger_ret_1d", "SBUX_zscore_60d", "EWY_Korea_zscore_60d", "BA_ret_1d", "CMCSA_ret_1d", "NWL_Newell_ret_20d", "GILD_Gilead_ret_20d", "AXP_Amex_ret_20d", "3M_ret_5d", "IYR_US_REIT2_zscore_60d"], "is_new": true}, {"model_id": "new_h1_CALM_XGBoost_N15_t1", "algo": "XGBoost", "regime": "CALM", "horizon": 1, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "US1Y_Rate_ret_20d", "Nikkei_Japan_vol_20d", "US30Y_Rate_ret_20d", "AMD_ret_1d", "CTAS_Cintas_vol_20d", "Brent_Oil_FRED_ret_5d", "DOW_Price_zscore_60d", "TXN_vol_20d", "ASX_Australia_vol_20d", "MO_AltriaMG_ret_1d", "US7Y_Rate_ret_20d", "DIS_vol_20d", "Nikkei_Japan_zscore_60d"], "is_new": true}, {"model_id": "new_h1_CALM_XGBoost_N15_t2", "algo": "XGBoost", "regime": "CALM", "horizon": 1, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "AVB_AvalonBay_zscore_60d", "LOW_Lowes_ret_20d", "EWH_HongKong_ret_5d", "EWY_Korea_ret_20d", "vix_acceleration_1d", "CPB_CampbellSoup_vol_20d", "LMT_LockheedMartin_vol_20d", "US30Y_Rate_ret_20d", "NEE_NextEra_ret_20d", "XLY_Disc_vol_20d", "EFFR_ret_1d", "EWQ_France_ret_20d", "EMR_Emerson_ret_20d"], "is_new": true}, {"model_id": "new_h1_CALM_XGBoost_N15_t3", "algo": "XGBoost", "regime": "CALM", "horizon": 1, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "IYR_US_REIT2_zscore_60d", "NFCI_ret_5d", "vix_mean_abs_ret_5d", "EWY_Korea_ret_20d", "GD_GeneralDynamics_zscore_60d", "EFFR_vol_20d", "CPB_CampbellSoup_zscore_60d", "MSTR_Bitcoin3_ret_20d", "ASX_Australia_vol_20d", "PPL_PPL_ret_1d", "EWL_Switzerland_zscore_60d", "EWG_Germany_ret_20d", "PFE_ret_1d"], "is_new": true}, {"model_id": "new_h1_CALM_XGBoost_N15_t4", "algo": "XGBoost", "regime": "CALM", "horizon": 1, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "AMZN_ret_5d", "spx_momentum_3d", "EWA_Australia_zscore_60d", "M_Macys_vol_20d", "IYM_BasicMaterials_ret_20d", "HD_ret_1d", "BTI_BritishAmerican_ret_5d", "LMT_LockheedMartin_ret_1d", "heston_var_ev_h5", "SLB_Schlumberger_ret_5d", "TM_Telephone_ret_1d", "AORD_AUS_zscore_60d", "Core_CPI_zscore_60d"], "is_new": true}, {"model_id": "new_h1_CALM_XGBoost_N15_t5", "algo": "XGBoost", "regime": "CALM", "horizon": 1, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "CPB_CampbellSoup_zscore_60d", "VVIX_ret_20d", "ORCL_vol_20d", "LLY_zscore_60d", "INTC_ret_1d", "XLF_Fin_vol_20d", "SLB_Schlumberger_ret_5d", "Core_CPI_zscore_60d", "MSTR_Bitcoin3_ret_5d", "US5Y_Rate_ret_5d", "EWA_Australia_ret_1d", "EXC_Exelon_ret_1d", "AXP_Amex_ret_20d"], "is_new": true}, {"model_id": "new_h1_CALM_XGBoost_N15_t6", "algo": "XGBoost", "regime": "CALM", "horizon": 1, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "LUV_SouthwestAir_ret_5d", "Nikkei_Japan_zscore_60d", "BTI_BritishAmerican_ret_5d", "vix_acceleration_1d", "IYM_BasicMaterials_ret_20d", "XOM_ret_1d", "INTC_ret_1d", "spx_vol_5d", "DAX_Germany_vol_20d", "HangSeng_HK_ret_1d", "SLB_Schlumberger_ret_1d", "IBEX_Spain_ret_20d", "NEE_NextEra_ret_20d"], "is_new": true}, {"model_id": "new_h1_CALM_XGBoost_N15_t7", "algo": "XGBoost", "regime": "CALM", "horizon": 1, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWA_Australia_zscore_60d", "NVDA_vol_20d", "vix_acceleration_1d", "MSTR_Bitcoin3_ret_20d", "EOG_EOGResources_vol_20d", "XLV_Health_zscore_60d", "US3M_Rate_zscore_60d", "HangSeng_HK_ret_5d", "Michigan_Sentiment_ret_20d", "T_ret_1d", "3M_ret_5d", "HD_ret_5d", "EWH_HongKong_ret_5d"], "is_new": true}, {"model_id": "new_h1_CALM_XGBoost_N20_t0", "algo": "XGBoost", "regime": "CALM", "horizon": 1, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "PAYX_Paychex_ret_20d", "EWJ_Japan_vol_20d", "SLB_Schlumberger_ret_5d", "QQQ_vol_20d", "EOG_EOGResources_ret_5d", "spx_momentum_3d", "CI_Cigna_vol_20d", "HD_ret_20d", "SO_SouthernCo_ret_5d", "spx_vol_5d", "heston_ev_h3", "XLY_Disc_vol_20d", "IYR_US_REIT2_zscore_60d", "CMCSA_ret_1d", "EWQ_France_ret_20d", "IBEX_Spain_ret_20d", "EWM_Malaysia_ret_1d", "EWC_Canada_zscore_60d"], "is_new": true}, {"model_id": "new_h1_CALM_XGBoost_N20_t1", "algo": "XGBoost", "regime": "CALM", "horizon": 1, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "QQQ_vol_20d", "XLK_Tech_zscore_60d", "Brent_Oil_FRED_ret_20d", "PCAR_PaccarInc_ret_5d", "NOC_Northrop_ret_20d", "gjr_condvar_h1", "US5Y_Rate_ret_5d", "EWA_Australia_zscore_60d", "MSTR_Bitcoin3_ret_5d", "SCHW_Schwab_ret_5d", "M_Macys_vol_20d", "EWS_Singapore_ret_5d", "DIS_vol_20d", "spx_abs_ret_max_5d", "3M_vol_20d", "CPB_CampbellSoup_zscore_60d", "NWL_Newell_ret_20d", "EWL_Switzerland_zscore_60d"], "is_new": true}, {"model_id": "new_h1_CALM_XGBoost_N20_t2", "algo": "XGBoost", "regime": "CALM", "horizon": 1, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "DAX_Germany_vol_20d", "PCAR_PaccarInc_ret_5d", "DE_Deere_vol_20d", "PG_ret_20d", "vix_acceleration_1d", "TED_Spread_zscore_60d", "US30Y_Rate_ret_20d", "VVIX_ret_20d", "INTC_ret_5d", "PAYX_Paychex_ret_20d", "XLK_Tech_zscore_60d", "SLB_Schlumberger_ret_1d", "Nikkei_Japan_vol_20d", "T_ret_1d", "PAYX_Paychex_vol_20d", "EFFR_vol_20d", "QQQ_vol_20d", "PAYX_Paychex_zscore_60d"], "is_new": true}, {"model_id": "new_h1_CALM_XGBoost_N20_t3", "algo": "XGBoost", "regime": "CALM", "horizon": 1, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "XOM_ret_20d", "EWQ_France_zscore_60d", "GILD_Gilead_ret_20d", "ORCL_zscore_60d", "Brent_Oil_FRED_ret_5d", "HD_ret_5d", "EWQ_France_ret_20d", "GE_ret_1d", "IYR_US_REIT2_zscore_60d", "INTC_ret_5d", "TM_Telephone_vol_20d", "CCI_CrownCastle_vol_20d", "DHR_ret_1d", "DOW_Price_zscore_60d", "EWM_Malaysia_zscore_60d", "MSTR_Bitcoin3_ret_5d", "EFFR_ret_1d", "EWJ_Japan_vol_20d"], "is_new": true}, {"model_id": "new_h1_CALM_XGBoost_N20_t4", "algo": "XGBoost", "regime": "CALM", "horizon": 1, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "TM_Telephone_vol_20d", "PLD_Prologis_ret_5d", "NOC_Northrop_ret_20d", "BLK_BlackRock_zscore_60d", "EWG_Germany_vol_20d", "NVDA_vol_20d", "MS_MorganStanley_ret_1d", "US7Y_Rate_ret_20d", "AMT_AmericanTower_ret_1d", "US5Y_Rate_ret_5d", "MSTR_Bitcoin3_ret_20d", "Retail_Sales_zscore_60d", "3M_vol_20d", "LMT_LockheedMartin_vol_20d", "HangSeng_HK_ret_1d", "TM_Telephone_ret_1d", "TXN_vol_20d", "XLB_Materials_zscore_60d"], "is_new": true}, {"model_id": "new_h1_CALM_XGBoost_N20_t5", "algo": "XGBoost", "regime": "CALM", "horizon": 1, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "ORCL_vol_20d", "SBUX_vol_20d", "AORD_AUS_zscore_60d", "MS_MorganStanley_ret_1d", "Industrial_Production_zscore_60d", "vix_mean_abs_ret_5d", "Michigan_Sentiment_ret_20d", "ENB_EnbridgeInc_ret_1d", "PLD_Prologis_ret_5d", "DAX_Germany_zscore_60d", "XOM_ret_1d", "TGT_Target_zscore_60d", "AMZN_ret_5d", "MS_MorganStanley_ret_5d", "VRP_ma5", "US3M_Rate_vol_20d", "spx_vol_5d", "CI_Cigna_vol_20d"], "is_new": true}, {"model_id": "new_h1_CALM_XGBoost_N20_t6", "algo": "XGBoost", "regime": "CALM", "horizon": 1, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EQR_Equity_ret_1d", "AXP_Amex_vol_20d", "EXC_Exelon_zscore_60d", "MSTR_Bitcoin3_ret_20d", "IWM_SmallCap_vol_20d", "EWJ_Japan_vol_20d", "PCAR_PaccarInc_ret_5d", "Industrial_Production_zscore_60d", "HD_ret_1d", "NVDA_vol_20d", "EWY_Korea_zscore_60d", "CCI_CrownCastle_vol_20d", "PPL_PPL_ret_1d", "AMGN_Amgen_ret_1d", "TM_Telephone_vol_20d", "BA_ret_1d", "LMT_LockheedMartin_ret_1d", "EWM_Malaysia_ret_1d"], "is_new": true}, {"model_id": "new_h1_CALM_XGBoost_N20_t7", "algo": "XGBoost", "regime": "CALM", "horizon": 1, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWY_Korea_ret_20d", "EWG_Germany_ret_20d", "AMT_AmericanTower_ret_1d", "XOM_ret_1d", "EFFR_ret_1d", "EWQ_France_zscore_60d", "DAX_Germany_zscore_60d", "QQQ_vol_20d", "vix_mean_abs_ret_5d", "Core_CPI_zscore_60d", "US3M_Rate_vol_20d", "EWL_Switzerland_vol_20d", "PLD_Prologis_ret_5d", "EXC_Exelon_ret_1d", "MO_AltriaMG_ret_1d", "ENB_EnbridgeInc_ret_1d", "XLF_Fin_vol_20d", "EMR_Emerson_ret_20d"], "is_new": true}, {"model_id": "new_h1_CALM_XGBoost_N25_t0", "algo": "XGBoost", "regime": "CALM", "horizon": 1, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "US5Y_Rate_ret_5d", "MSTR_Bitcoin3_ret_1d", "MS_MorganStanley_zscore_60d", "EWC_Canada_zscore_60d", "Brent_Oil_FRED_ret_5d", "VRP_ma5", "XOM_ret_20d", "EWS_Singapore_ret_5d", "CPB_CampbellSoup_vol_20d", "HangSeng_HK_ret_5d", "WTI_Oil_FRED_zscore_60d", "Core_PCE_zscore_60d", "US3M_Rate_vol_20d", "AVB_AvalonBay_zscore_60d", "LLY_zscore_60d", "TGT_Target_zscore_60d", "Nikkei_Japan_vol_20d", "MO_AltriaMG_ret_1d", "BTI_BritishAmerican_ret_5d", "SLB_Schlumberger_ret_1d", "JNJ_ret_1d", "AMD_ret_5d", "NVDA_vol_20d"], "is_new": true}, {"model_id": "new_h1_CALM_XGBoost_N25_t1", "algo": "XGBoost", "regime": "CALM", "horizon": 1, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "heston_ev_h3", "PFE_ret_1d", "EFFR_ret_1d", "ORCL_vol_20d", "EQIX_Equinix_ret_5d", "T10Y2Y_Spread_ret_5d", "GE_ret_1d", "XLV_Health_zscore_60d", "TM_Telephone_ret_1d", "SBUX_ret_5d", "PLD_Prologis_ret_5d", "spx_momentum_3d", "VRP_ma5", "GD_GeneralDynamics_zscore_60d", "ITT_ITTInc_ret_5d", "EWS_Singapore_ret_5d", "CLX_Clorox_vol_20d", "HD_ret_1d", "PAYX_Paychex_zscore_60d", "XLK_Tech_zscore_60d", "PG_ret_20d", "US3M_Rate_zscore_60d", "MSTR_Bitcoin3_ret_20d"], "is_new": true}, {"model_id": "new_h1_CALM_XGBoost_N25_t2", "algo": "XGBoost", "regime": "CALM", "horizon": 1, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EQR_Equity_ret_1d", "SBUX_vol_20d", "EWJ_Japan_vol_20d", "LMT_LockheedMartin_vol_20d", "EXC_Exelon_zscore_60d", "JNJ_ret_1d", "HD_ret_1d", "VVIX_ret_20d", "GILD_Gilead_ret_20d", "US1Y_Rate_ret_5d", "AMD_ret_1d", "SPY_zscore_60d", "EWA_Australia_ret_1d", "CI_Cigna_vol_20d", "EWQ_France_zscore_60d", "NEE_NextEra_ret_20d", "SBUX_zscore_60d", "heston_ev_h3", "XLV_Health_zscore_60d", "QQQ_vol_20d", "XLF_Fin_vol_20d", "PAYX_Paychex_vol_20d", "AMGN_Amgen_ret_1d"], "is_new": true}, {"model_id": "new_h1_CALM_XGBoost_N25_t3", "algo": "XGBoost", "regime": "CALM", "horizon": 1, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWS_Singapore_ret_5d", "HangSeng_HK_vol_20d", "spx_vol_5d", "US6M_Rate_ret_20d", "MO_AltriaMG_ret_1d", "QQQ_vol_20d", "US1Y_Rate_ret_20d", "TED_Spread_zscore_60d", "LLY_zscore_60d", "NWL_Newell_ret_20d", "T_ret_1d", "IYM_BasicMaterials_ret_20d", "LMT_LockheedMartin_vol_20d", "PCAR_PaccarInc_ret_5d", "GE_ret_1d", "vix_acceleration_1d", "EWY_Korea_zscore_60d", "EWM_Malaysia_zscore_60d", "Core_PCE_zscore_60d", "EWA_Australia_zscore_60d", "INTC_ret_1d", "WTI_Oil_FRED_zscore_60d", "TM_Telephone_ret_1d"], "is_new": true}, {"model_id": "new_h1_CALM_XGBoost_N25_t4", "algo": "XGBoost", "regime": "CALM", "horizon": 1, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "SBUX_ret_5d", "GILD_Gilead_ret_20d", "VOD_Vodafone_zscore_60d", "TED_Spread_vol_20d", "VVIX_ret_20d", "HD_ret_20d", "CPB_CampbellSoup_ret_5d", "EWA_Australia_zscore_60d", "HUM_Humana_ret_5d", "EWM_Malaysia_vol_20d", "LOW_Lowes_ret_5d", "EQIX_Equinix_ret_5d", "HangSeng_HK_ret_5d", "AMD_ret_5d", "TGT_Target_zscore_60d", "DAX_Germany_zscore_60d", "vix_acceleration_1d", "AXP_Amex_ret_20d", "BA_ret_1d", "US1Y_Rate_ret_20d", "HD_zscore_60d", "XLK_Tech_zscore_60d", "SBUX_zscore_60d"], "is_new": true}, {"model_id": "new_h1_CALM_XGBoost_N25_t5", "algo": "XGBoost", "regime": "CALM", "horizon": 1, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "Retail_Sales_zscore_60d", "ORCL_zscore_60d", "JNJ_ret_1d", "ES_Evergy_ret_1d", "BA_ret_1d", "EWY_Korea_zscore_60d", "vix_mean_abs_ret_5d", "Core_CPI_zscore_60d", "BTI_BritishAmerican_ret_20d", "M_Macys_vol_20d", "SBUX_zscore_60d", "Brent_Oil_FRED_ret_20d", "US5Y_Rate_ret_5d", "spx_abs_ret_max_5d", "DOW_Price_zscore_60d", "XLV_Health_zscore_60d", "US3M_Rate_vol_20d", "HangSeng_HK_ret_5d", "CI_Cigna_vol_20d", "EWY_Korea_ret_20d", "Core_PCE_zscore_60d", "US1Y_Rate_ret_5d", "NOC_Northrop_ret_20d"], "is_new": true}, {"model_id": "new_h1_CALM_XGBoost_N25_t6", "algo": "XGBoost", "regime": "CALM", "horizon": 1, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "PAYX_Paychex_ret_20d", "HangSeng_HK_ret_1d", "VOD_Vodafone_zscore_60d", "Nikkei_Japan_vol_20d", "HangSeng_HK_vol_20d", "US3M_Rate_zscore_60d", "XLY_Disc_vol_20d", "BLK_BlackRock_zscore_60d", "XLK_Tech_zscore_60d", "CPB_CampbellSoup_ret_5d", "EWH_HongKong_ret_5d", "XOM_ret_20d", "EWJ_Japan_vol_20d", "FedFunds_zscore_60d", "CCI_CrownCastle_vol_20d", "EFFR_vol_20d", "EWQ_France_zscore_60d", "EOG_EOGResources_vol_20d", "US1Y_Rate_ret_20d", "Brent_Oil_FRED_ret_5d", "heston_var_ev_h3", "3M_ret_5d", "NEE_NextEra_ret_20d"], "is_new": true}, {"model_id": "new_h1_CALM_XGBoost_N25_t7", "algo": "XGBoost", "regime": "CALM", "horizon": 1, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWM_Malaysia_zscore_60d", "HD_ret_5d", "HD_ret_20d", "PCAR_PaccarInc_ret_5d", "Brent_Oil_FRED_ret_20d", "vix_mean_abs_ret_5d", "FedFunds_zscore_60d", "XOM_ret_20d", "VOD_Vodafone_zscore_60d", "EWC_Canada_zscore_60d", "NWL_Newell_ret_20d", "Core_CPI_zscore_60d", "TXN_vol_20d", "EWM_Malaysia_ret_1d", "DOW_Price_zscore_60d", "GD_GeneralDynamics_zscore_60d", "DHR_vol_20d", "spx_abs_ret_max_5d", "BLK_BlackRock_zscore_60d", "ENB_EnbridgeInc_ret_1d", "HangSeng_HK_vol_20d", "US3M_Rate_zscore_60d", "vix_acceleration_1d"], "is_new": true}, {"model_id": "new_h1_CALM_XGBoost_N30_t0", "algo": "XGBoost", "regime": "CALM", "horizon": 1, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "Michigan_Sentiment_ret_20d", "GE_ret_1d", "SJM_JM_Smucker_ret_5d", "TM_Telephone_vol_20d", "SCHW_Schwab_ret_5d", "ORCL_zscore_60d", "CCI_CrownCastle_vol_20d", "WTI_Oil_FRED_zscore_60d", "AVB_AvalonBay_zscore_60d", "Nikkei_Japan_zscore_60d", "ITT_ITTInc_ret_5d", "XLB_Materials_zscore_60d", "EWY_Korea_zscore_60d", "AMZN_ret_5d", "CMCSA_ret_1d", "EWM_Malaysia_vol_20d", "US5Y_Rate_ret_5d", "VOD_Vodafone_zscore_60d", "AMT_AmericanTower_ret_1d", "GD_GeneralDynamics_zscore_60d", "XLF_Fin_vol_20d", "heston_var_ev_h3", "EOG_EOGResources_vol_20d", "PPL_PPL_ret_1d", "SPY_zscore_60d", "PAYX_Paychex_ret_20d", "HD_ret_1d", "EQR_Equity_ret_1d"], "is_new": true}, {"model_id": "new_h1_CALM_XGBoost_N30_t1", "algo": "XGBoost", "regime": "CALM", "horizon": 1, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "PPL_PPL_ret_1d", "LMT_LockheedMartin_ret_1d", "NEE_NextEra_ret_20d", "IYR_US_REIT2_zscore_60d", "ITT_ITTInc_ret_5d", "CPB_CampbellSoup_vol_20d", "EMR_Emerson_ret_20d", "IYM_BasicMaterials_ret_20d", "NFCI_ret_5d", "SCHW_Schwab_ret_5d", "HD_ret_5d", "M_Macys_vol_20d", "T_ret_1d", "ASX_Australia_vol_20d", "PCAR_PaccarInc_ret_5d", "CLX_Clorox_vol_20d", "INTC_ret_1d", "EWY_Korea_zscore_60d", "spx_abs_ret_max_5d", "Core_CPI_zscore_60d", "Michigan_Sentiment_ret_20d", "SO_SouthernCo_ret_5d", "3M_ret_5d", "TM_Telephone_vol_20d", "PG_ret_20d", "heston_var_ev_h3", "BA_ret_1d", "EWA_Australia_ret_1d"], "is_new": true}, {"model_id": "new_h1_CALM_XGBoost_N30_t2", "algo": "XGBoost", "regime": "CALM", "horizon": 1, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "BA_ret_1d", "EWG_Germany_vol_20d", "GD_GeneralDynamics_zscore_60d", "T_ret_1d", "DE_Deere_ret_5d", "HD_ret_5d", "ORCL_zscore_60d", "SBUX_vol_20d", "DAX_Germany_vol_20d", "AORD_AUS_zscore_60d", "gjr_condvar_h1", "XLF_Fin_vol_20d", "EWQ_France_zscore_60d", "EFFR_vol_20d", "IBEX_Spain_ret_20d", "US1Y_Rate_ret_20d", "AMD_ret_5d", "XLK_Tech_zscore_60d", "NEE_NextEra_ret_20d", "CMCSA_ret_1d", "EWM_Malaysia_ret_1d", "Core_CPI_zscore_60d", "PG_ret_20d", "Core_PCE_zscore_60d", "EFFR_ret_1d", "SJM_JM_Smucker_ret_1d", "HD_zscore_60d", "SLB_Schlumberger_ret_5d"], "is_new": true}, {"model_id": "new_h1_CALM_XGBoost_N30_t3", "algo": "XGBoost", "regime": "CALM", "horizon": 1, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "PPL_PPL_ret_1d", "EWS_Singapore_ret_5d", "PCAR_PaccarInc_ret_5d", "US1Y_Rate_ret_20d", "DIS_vol_20d", "MSTR_Bitcoin3_ret_5d", "EWA_Australia_ret_1d", "WTI_Oil_FRED_zscore_60d", "SLB_Schlumberger_ret_5d", "DE_Deere_vol_20d", "BTI_BritishAmerican_ret_5d", "GILD_Gilead_ret_20d", "NOC_Northrop_ret_20d", "JNJ_ret_1d", "SLB_Schlumberger_ret_1d", "GD_GeneralDynamics_zscore_60d", "3M_ret_5d", "HD_ret_20d", "HangSeng_HK_ret_1d", "VOD_Vodafone_zscore_60d", "US6M_Rate_ret_20d", "SBUX_zscore_60d", "DOW_Price_zscore_60d", "PAYX_Paychex_vol_20d", "TM_Telephone_ret_1d", "SBUX_ret_5d", "LLY_zscore_60d", "IYR_US_REIT2_zscore_60d"], "is_new": true}, {"model_id": "new_h1_CALM_XGBoost_N30_t4", "algo": "XGBoost", "regime": "CALM", "horizon": 1, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWA_Australia_ret_1d", "EWQ_France_zscore_60d", "US3Y_Rate_ret_5d", "MSTR_Bitcoin3_ret_5d", "EWM_Malaysia_zscore_60d", "NFCI_ret_5d", "AXP_Amex_ret_20d", "INTC_ret_5d", "QQQ_vol_20d", "MSTR_Bitcoin3_ret_20d", "spx_momentum_3d", "DOW_Price_zscore_60d", "SCHW_Schwab_ret_5d", "ASX_Australia_vol_20d", "DAX_Germany_zscore_60d", "SJM_JM_Smucker_ret_5d", "VRP_ma5", "IYM_BasicMaterials_ret_20d", "CPB_CampbellSoup_ret_5d", "ORCL_zscore_60d", "WTI_Oil_FRED_zscore_60d", "TED_Spread_vol_20d", "PCAR_PaccarInc_ret_5d", "AMD_ret_5d", "DHR_vol_20d", "NEE_NextEra_ret_20d", "EFFR_ret_1d", "Brent_Oil_FRED_ret_5d"], "is_new": true}, {"model_id": "new_h1_CALM_XGBoost_N30_t5", "algo": "XGBoost", "regime": "CALM", "horizon": 1, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "hmm_p_stress", "CCI_CrownCastle_vol_20d", "IYM_BasicMaterials_ret_20d", "TGT_Target_zscore_60d", "MSTR_Bitcoin3_ret_20d", "INTC_ret_5d", "XOM_ret_20d", "SBUX_vol_20d", "TM_Telephone_vol_20d", "ASX_Australia_vol_20d", "XLY_Disc_vol_20d", "BTI_BritishAmerican_ret_5d", "EWQ_France_ret_20d", "HD_ret_5d", "EWM_Malaysia_vol_20d", "CMCSA_ret_1d", "ORCL_vol_20d", "CPB_CampbellSoup_ret_5d", "XLB_Materials_zscore_60d", "PAYX_Paychex_ret_20d", "ENB_EnbridgeInc_ret_1d", "HangSeng_HK_ret_1d", "AORD_AUS_zscore_60d", "AXP_Amex_ret_20d", "IWM_SmallCap_vol_20d", "BDX_Becton_Dickinson_ret_20d", "HUM_Humana_ret_5d", "LOW_Lowes_ret_20d"], "is_new": true}, {"model_id": "new_h1_CALM_XGBoost_N30_t6", "algo": "XGBoost", "regime": "CALM", "horizon": 1, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "TED_Spread_zscore_60d", "NFCI_ret_5d", "US3M_Rate_vol_20d", "EQR_Equity_ret_1d", "EOG_EOGResources_ret_5d", "EOG_EOGResources_vol_20d", "CCI_CrownCastle_vol_20d", "XLK_Tech_zscore_60d", "MSTR_Bitcoin3_ret_20d", "XLY_Disc_vol_20d", "EWQ_France_zscore_60d", "SJM_JM_Smucker_ret_1d", "US7Y_Rate_ret_20d", "DHR_vol_20d", "EWM_Malaysia_zscore_60d", "PG_ret_20d", "EXC_Exelon_ret_1d", "PFE_ret_1d", "XLF_Fin_vol_20d", "AMD_ret_1d", "PLD_Prologis_ret_5d", "HD_ret_5d", "TXN_vol_20d", "EFFR_vol_20d", "Brent_Oil_FRED_ret_20d", "PAYX_Paychex_ret_20d", "LLY_zscore_60d", "HangSeng_HK_ret_1d"], "is_new": true}, {"model_id": "new_h1_CALM_XGBoost_N30_t7", "algo": "XGBoost", "regime": "CALM", "horizon": 1, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "HangSeng_HK_ret_5d", "EWC_Canada_zscore_60d", "MSTR_Bitcoin3_ret_5d", "SBUX_ret_5d", "CPB_CampbellSoup_zscore_60d", "EWQ_France_ret_20d", "MSTR_Bitcoin3_ret_20d", "US3Y_Rate_ret_5d", "LOW_Lowes_ret_20d", "US5Y_Rate_ret_5d", "HUM_Humana_ret_5d", "gjr_condvar_h1", "AMT_AmericanTower_ret_1d", "MS_MorganStanley_zscore_60d", "GILD_Gilead_ret_20d", "AMD_ret_1d", "US3M_Rate_vol_20d", "AXP_Amex_vol_20d", "SJM_JM_Smucker_ret_1d", "HD_ret_5d", "EWM_Malaysia_vol_20d", "LMT_LockheedMartin_vol_20d", "EWL_Switzerland_zscore_60d", "EWA_Australia_ret_1d", "EWY_Korea_zscore_60d", "DHR_ret_1d", "CPB_CampbellSoup_ret_20d", "AVB_AvalonBay_zscore_60d"], "is_new": true}, {"model_id": "new_h1_CALM_LightGBM_N5_t0", "algo": "LightGBM", "regime": "CALM", "horizon": 1, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "DE_Deere_ret_5d", "CMCSA_ret_1d", "HUM_Humana_ret_5d"], "is_new": true}, {"model_id": "new_h1_CALM_LightGBM_N5_t1", "algo": "LightGBM", "regime": "CALM", "horizon": 1, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "DE_Deere_ret_5d", "TM_Telephone_vol_20d", "LUV_SouthwestAir_ret_5d"], "is_new": true}, {"model_id": "new_h1_CALM_LightGBM_N5_t2", "algo": "LightGBM", "regime": "CALM", "horizon": 1, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "SBUX_ret_5d", "HUM_Humana_ret_5d", "Industrial_Production_zscore_60d"], "is_new": true}, {"model_id": "new_h1_CALM_LightGBM_N5_t3", "algo": "LightGBM", "regime": "CALM", "horizon": 1, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWM_Malaysia_ret_1d", "MS_MorganStanley_zscore_60d", "US1Y_Rate_ret_5d"], "is_new": true}, {"model_id": "new_h1_CALM_LightGBM_N5_t4", "algo": "LightGBM", "regime": "CALM", "horizon": 1, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "FedFunds_zscore_60d", "TM_Telephone_ret_1d", "DAX_Germany_vol_20d"], "is_new": true}, {"model_id": "new_h1_CALM_LightGBM_N5_t5", "algo": "LightGBM", "regime": "CALM", "horizon": 1, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "heston_var_ev_h3", "MRK_Merck_zscore_60d", "CPB_CampbellSoup_zscore_60d"], "is_new": true}, {"model_id": "new_h1_CALM_LightGBM_N5_t6", "algo": "LightGBM", "regime": "CALM", "horizon": 1, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "XLY_Disc_vol_20d", "US30Y_Rate_ret_20d", "IWM_SmallCap_vol_20d"], "is_new": true}, {"model_id": "new_h1_CALM_LightGBM_N5_t7", "algo": "LightGBM", "regime": "CALM", "horizon": 1, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "HangSeng_HK_ret_5d", "hmm_p_stress", "NEE_NextEra_ret_20d"], "is_new": true}, {"model_id": "new_h1_CALM_LightGBM_N8_t0", "algo": "LightGBM", "regime": "CALM", "horizon": 1, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "HangSeng_HK_ret_5d", "SCHW_Schwab_ret_5d", "Core_CPI_zscore_60d", "CPB_CampbellSoup_ret_20d", "EWQ_France_ret_20d", "TM_Telephone_ret_1d"], "is_new": true}, {"model_id": "new_h1_CALM_LightGBM_N8_t1", "algo": "LightGBM", "regime": "CALM", "horizon": 1, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "vix_mean_abs_ret_5d", "TM_Telephone_vol_20d", "NVDA_vol_20d", "VOD_Vodafone_zscore_60d", "EFFR_ret_1d", "NOC_Northrop_ret_20d"], "is_new": true}, {"model_id": "new_h1_CALM_LightGBM_N8_t2", "algo": "LightGBM", "regime": "CALM", "horizon": 1, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "HD_ret_1d", "ASX_Australia_ret_5d", "MO_AltriaMG_ret_1d", "EWL_Switzerland_vol_20d", "PAYX_Paychex_ret_20d", "EWQ_France_ret_20d"], "is_new": true}, {"model_id": "new_h1_CALM_LightGBM_N8_t3", "algo": "LightGBM", "regime": "CALM", "horizon": 1, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "BLK_BlackRock_zscore_60d", "MS_MorganStanley_zscore_60d", "GILD_Gilead_ret_20d", "EWA_Australia_ret_1d", "DAX_Germany_vol_20d", "PCAR_PaccarInc_ret_5d"], "is_new": true}, {"model_id": "new_h1_CALM_LightGBM_N8_t4", "algo": "LightGBM", "regime": "CALM", "horizon": 1, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "IYR_US_REIT2_zscore_60d", "EWS_Singapore_ret_5d", "DAX_Germany_zscore_60d", "US3M_Rate_vol_20d", "EWG_Germany_vol_20d", "BTI_BritishAmerican_ret_5d"], "is_new": true}, {"model_id": "new_h1_CALM_LightGBM_N8_t5", "algo": "LightGBM", "regime": "CALM", "horizon": 1, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "IYM_BasicMaterials_ret_20d", "DIS_vol_20d", "VVIX_ret_20d", "3M_ret_5d", "GILD_Gilead_ret_20d", "DE_Deere_ret_5d"], "is_new": true}, {"model_id": "new_h1_CALM_LightGBM_N8_t6", "algo": "LightGBM", "regime": "CALM", "horizon": 1, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "spx_vol_5d", "AMZN_ret_5d", "DE_Deere_ret_5d", "AMD_ret_5d", "LOW_Lowes_ret_20d", "EWQ_France_ret_20d"], "is_new": true}, {"model_id": "new_h1_CALM_LightGBM_N8_t7", "algo": "LightGBM", "regime": "CALM", "horizon": 1, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "FedFunds_zscore_60d", "BA_ret_1d", "DAX_Germany_zscore_60d", "NVDA_vol_20d", "TED_Spread_vol_20d", "Core_CPI_zscore_60d"], "is_new": true}, {"model_id": "new_h1_CALM_LightGBM_N10_t0", "algo": "LightGBM", "regime": "CALM", "horizon": 1, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "LOW_Lowes_ret_5d", "SO_SouthernCo_ret_5d", "VRP_ma5", "CPB_CampbellSoup_zscore_60d", "PAYX_Paychex_zscore_60d", "SJM_JM_Smucker_ret_5d", "EWL_Switzerland_zscore_60d", "XLV_Health_zscore_60d"], "is_new": true}, {"model_id": "new_h1_CALM_LightGBM_N10_t1", "algo": "LightGBM", "regime": "CALM", "horizon": 1, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "BLK_BlackRock_zscore_60d", "Retail_Sales_zscore_60d", "M_Macys_vol_20d", "Nikkei_Japan_vol_20d", "CPB_CampbellSoup_zscore_60d", "SBUX_zscore_60d", "PAYX_Paychex_ret_20d", "Brent_Oil_FRED_ret_20d"], "is_new": true}, {"model_id": "new_h1_CALM_LightGBM_N10_t2", "algo": "LightGBM", "regime": "CALM", "horizon": 1, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "DAX_Germany_zscore_60d", "GILD_Gilead_ret_20d", "XOM_ret_1d", "heston_ev_h3", "US7Y_Rate_ret_20d", "T_ret_1d", "IWM_SmallCap_vol_20d", "MSTR_Bitcoin3_ret_20d"], "is_new": true}, {"model_id": "new_h1_CALM_LightGBM_N10_t3", "algo": "LightGBM", "regime": "CALM", "horizon": 1, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "CPB_CampbellSoup_ret_5d", "Core_CPI_zscore_60d", "spx_abs_ret_max_5d", "hmm_p_stress", "US1Y_Rate_ret_20d", "EWL_Switzerland_vol_20d", "Nikkei_Japan_zscore_60d", "PAYX_Paychex_zscore_60d"], "is_new": true}, {"model_id": "new_h1_CALM_LightGBM_N10_t4", "algo": "LightGBM", "regime": "CALM", "horizon": 1, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "M_Macys_vol_20d", "NOC_Northrop_ret_20d", "CPB_CampbellSoup_ret_20d", "INTC_ret_1d", "Nikkei_Japan_vol_20d", "vix_acceleration_1d", "EXC_Exelon_ret_1d", "MRK_Merck_zscore_60d"], "is_new": true}, {"model_id": "new_h1_CALM_LightGBM_N10_t5", "algo": "LightGBM", "regime": "CALM", "horizon": 1, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "CI_Cigna_vol_20d", "CMCSA_ret_1d", "Nikkei_Japan_zscore_60d", "CPB_CampbellSoup_ret_20d", "MRK_Merck_zscore_60d", "DE_Deere_ret_5d", "US1Y_Rate_ret_5d", "BLK_BlackRock_zscore_60d"], "is_new": true}, {"model_id": "new_h1_CALM_LightGBM_N10_t6", "algo": "LightGBM", "regime": "CALM", "horizon": 1, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "LMT_LockheedMartin_vol_20d", "vix_acceleration_1d", "NWL_Newell_ret_20d", "BA_ret_1d", "HD_ret_5d", "WTI_Oil_FRED_zscore_60d", "IBEX_Spain_ret_20d", "CPB_CampbellSoup_vol_20d"], "is_new": true}, {"model_id": "new_h1_CALM_LightGBM_N10_t7", "algo": "LightGBM", "regime": "CALM", "horizon": 1, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWM_Malaysia_ret_1d", "CPB_CampbellSoup_zscore_60d", "LMT_LockheedMartin_ret_1d", "US1Y_Rate_ret_5d", "MO_AltriaMG_ret_1d", "Core_CPI_zscore_60d", "HD_ret_1d", "US3Y_Rate_ret_5d"], "is_new": true}, {"model_id": "new_h1_CALM_LightGBM_N12_t0", "algo": "LightGBM", "regime": "CALM", "horizon": 1, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "AORD_AUS_zscore_60d", "EQR_Equity_ret_1d", "GE_ret_1d", "IBEX_Spain_ret_20d", "Industrial_Production_zscore_60d", "LMT_LockheedMartin_ret_1d", "3M_vol_20d", "DHR_vol_20d", "US3M_Rate_vol_20d", "spx_momentum_3d"], "is_new": true}, {"model_id": "new_h1_CALM_LightGBM_N12_t1", "algo": "LightGBM", "regime": "CALM", "horizon": 1, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "CMCSA_ret_1d", "TED_Spread_zscore_60d", "EWL_Switzerland_vol_20d", "MSTR_Bitcoin3_ret_1d", "NVDA_vol_20d", "PCAR_PaccarInc_ret_5d", "JNJ_ret_1d", "EFFR_vol_20d", "MO_AltriaMG_ret_1d", "IBEX_Spain_ret_20d"], "is_new": true}, {"model_id": "new_h1_CALM_LightGBM_N12_t2", "algo": "LightGBM", "regime": "CALM", "horizon": 1, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EFFR_ret_1d", "EFFR_vol_20d", "XLY_Disc_vol_20d", "AXP_Amex_vol_20d", "CCI_CrownCastle_vol_20d", "GE_ret_1d", "PAYX_Paychex_ret_20d", "spx_vol_5d", "NWL_Newell_ret_20d", "SBUX_vol_20d"], "is_new": true}, {"model_id": "new_h1_CALM_LightGBM_N12_t3", "algo": "LightGBM", "regime": "CALM", "horizon": 1, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWL_Switzerland_vol_20d", "SO_SouthernCo_ret_5d", "Core_CPI_zscore_60d", "LLY_zscore_60d", "Nikkei_Japan_zscore_60d", "CTAS_Cintas_vol_20d", "HD_ret_20d", "VRP_ma5", "SBUX_vol_20d", "TED_Spread_vol_20d"], "is_new": true}, {"model_id": "new_h1_CALM_LightGBM_N12_t4", "algo": "LightGBM", "regime": "CALM", "horizon": 1, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "MS_MorganStanley_zscore_60d", "WTI_Oil_FRED_zscore_60d", "BDX_Becton_Dickinson_ret_20d", "3M_vol_20d", "EWG_Germany_vol_20d", "HangSeng_HK_ret_5d", "EWJ_Japan_vol_20d", "DHR_ret_1d", "EXC_Exelon_ret_1d", "HD_zscore_60d"], "is_new": true}, {"model_id": "new_h1_CALM_LightGBM_N12_t5", "algo": "LightGBM", "regime": "CALM", "horizon": 1, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "Nikkei_Japan_zscore_60d", "EOG_EOGResources_vol_20d", "DE_Deere_ret_5d", "MSTR_Bitcoin3_ret_20d", "TGT_Target_zscore_60d", "EWM_Malaysia_vol_20d", "SJM_JM_Smucker_ret_5d", "US30Y_Rate_ret_20d", "vix_mean_abs_ret_5d", "LMT_LockheedMartin_vol_20d"], "is_new": true}, {"model_id": "new_h1_CALM_LightGBM_N12_t6", "algo": "LightGBM", "regime": "CALM", "horizon": 1, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "AMGN_Amgen_ret_1d", "PFE_ret_1d", "Brent_Oil_FRED_ret_5d", "INTC_ret_1d", "US5Y_Rate_ret_5d", "Nikkei_Japan_vol_20d", "US3Y_Rate_ret_5d", "Michigan_Sentiment_ret_20d", "heston_var_ev_h3", "NOC_Northrop_ret_20d"], "is_new": true}, {"model_id": "new_h1_CALM_LightGBM_N12_t7", "algo": "LightGBM", "regime": "CALM", "horizon": 1, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "IWM_SmallCap_vol_20d", "EOG_EOGResources_ret_5d", "XLY_Disc_vol_20d", "PFE_ret_1d", "LOW_Lowes_ret_5d", "CLX_Clorox_vol_20d", "ORCL_vol_20d", "CPB_CampbellSoup_zscore_60d", "EWS_Singapore_ret_5d", "VOD_Vodafone_zscore_60d"], "is_new": true}, {"model_id": "new_h1_CALM_LightGBM_N15_t0", "algo": "LightGBM", "regime": "CALM", "horizon": 1, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "Industrial_Production_zscore_60d", "EWL_Switzerland_zscore_60d", "IYR_US_REIT2_zscore_60d", "EOG_EOGResources_ret_5d", "Brent_Oil_FRED_ret_5d", "SLB_Schlumberger_ret_5d", "heston_ev_h3", "Core_CPI_zscore_60d", "TED_Spread_zscore_60d", "EWA_Australia_zscore_60d", "LMT_LockheedMartin_ret_1d", "heston_var_ev_h7", "DIS_vol_20d"], "is_new": true}, {"model_id": "new_h1_CALM_LightGBM_N15_t1", "algo": "LightGBM", "regime": "CALM", "horizon": 1, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWC_Canada_zscore_60d", "EOG_EOGResources_vol_20d", "XLB_Materials_zscore_60d", "EFFR_ret_1d", "spx_abs_ret_max_5d", "Brent_Oil_FRED_ret_5d", "EQIX_Equinix_ret_5d", "XOM_ret_20d", "SCHW_Schwab_ret_5d", "PG_ret_20d", "US3M_Rate_vol_20d", "US6M_Rate_ret_20d", "PAYX_Paychex_vol_20d"], "is_new": true}, {"model_id": "new_h1_CALM_LightGBM_N15_t2", "algo": "LightGBM", "regime": "CALM", "horizon": 1, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "ORCL_zscore_60d", "BTI_BritishAmerican_ret_5d", "PPL_PPL_ret_1d", "PAYX_Paychex_ret_20d", "gjr_condvar_h1", "HangSeng_HK_ret_5d", "GILD_Gilead_ret_20d", "VRP_ma5", "BDX_Becton_Dickinson_ret_20d", "LMT_LockheedMartin_vol_20d", "SLB_Schlumberger_ret_1d", "Brent_Oil_FRED_ret_20d", "IBEX_Spain_ret_20d"], "is_new": true}, {"model_id": "new_h1_CALM_LightGBM_N15_t3", "algo": "LightGBM", "regime": "CALM", "horizon": 1, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "MO_AltriaMG_ret_1d", "HD_ret_5d", "Brent_Oil_FRED_ret_5d", "heston_var_ev_h7", "CMCSA_ret_1d", "XLV_Health_zscore_60d", "VVIX_ret_20d", "SLB_Schlumberger_ret_1d", "MSTR_Bitcoin3_ret_5d", "PG_ret_20d", "US6M_Rate_ret_20d", "US1Y_Rate_ret_20d", "AVB_AvalonBay_zscore_60d"], "is_new": true}, {"model_id": "new_h1_CALM_LightGBM_N15_t4", "algo": "LightGBM", "regime": "CALM", "horizon": 1, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "SJM_JM_Smucker_ret_5d", "DOW_Price_zscore_60d", "BLK_BlackRock_zscore_60d", "PAYX_Paychex_zscore_60d", "CPB_CampbellSoup_vol_20d", "JNJ_ret_1d", "CI_Cigna_vol_20d", "SLB_Schlumberger_ret_1d", "heston_var_ev_h7", "EOG_EOGResources_vol_20d", "AVB_AvalonBay_zscore_60d", "EWY_Korea_ret_20d", "ORCL_zscore_60d"], "is_new": true}, {"model_id": "new_h1_CALM_LightGBM_N15_t5", "algo": "LightGBM", "regime": "CALM", "horizon": 1, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "HD_ret_20d", "MS_MorganStanley_ret_1d", "NEE_NextEra_ret_20d", "Industrial_Production_zscore_60d", "WTI_Oil_FRED_zscore_60d", "BA_ret_1d", "gjr_condvar_h1", "heston_var_ev_h7", "IYR_US_REIT2_zscore_60d", "BTI_BritishAmerican_ret_20d", "IYM_BasicMaterials_ret_20d", "CPB_CampbellSoup_ret_20d", "US1Y_Rate_ret_5d"], "is_new": true}, {"model_id": "new_h1_CALM_LightGBM_N15_t6", "algo": "LightGBM", "regime": "CALM", "horizon": 1, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "spx_momentum_3d", "XLB_Materials_zscore_60d", "DOW_Price_zscore_60d", "IYR_US_REIT2_zscore_60d", "CCI_CrownCastle_vol_20d", "US7Y_Rate_ret_20d", "spx_vol_5d", "DHR_vol_20d", "LMT_LockheedMartin_ret_1d", "IBEX_Spain_ret_20d", "spx_abs_ret_max_5d", "MS_MorganStanley_ret_5d", "DE_Deere_ret_5d"], "is_new": true}, {"model_id": "new_h1_CALM_LightGBM_N15_t7", "algo": "LightGBM", "regime": "CALM", "horizon": 1, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "heston_var_ev_h5", "IYM_BasicMaterials_ret_20d", "DAX_Germany_zscore_60d", "vix_mean_abs_ret_5d", "TED_Spread_vol_20d", "CMCSA_ret_1d", "heston_var_ev_h7", "NWL_Newell_ret_20d", "SLB_Schlumberger_ret_5d", "LOW_Lowes_ret_5d", "AMGN_Amgen_ret_1d", "IBEX_Spain_ret_20d", "EQR_Equity_ret_1d"], "is_new": true}, {"model_id": "new_h1_CALM_LightGBM_N20_t0", "algo": "LightGBM", "regime": "CALM", "horizon": 1, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWM_Malaysia_ret_1d", "DHR_ret_1d", "EFFR_ret_1d", "Brent_Oil_FRED_ret_5d", "HUM_Humana_ret_5d", "CPB_CampbellSoup_ret_5d", "SLB_Schlumberger_ret_1d", "TM_Telephone_vol_20d", "TED_Spread_vol_20d", "ASX_Australia_vol_20d", "spx_vol_5d", "Industrial_Production_zscore_60d", "XLY_Disc_vol_20d", "DE_Deere_ret_5d", "VVIX_ret_20d", "MSTR_Bitcoin3_ret_1d", "EMR_Emerson_ret_20d", "EQR_Equity_ret_1d"], "is_new": true}, {"model_id": "new_h1_CALM_LightGBM_N20_t1", "algo": "LightGBM", "regime": "CALM", "horizon": 1, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "NFCI_ret_5d", "LUV_SouthwestAir_ret_5d", "EWQ_France_ret_20d", "PAYX_Paychex_ret_20d", "XLK_Tech_zscore_60d", "HD_ret_20d", "PAYX_Paychex_vol_20d", "EWM_Malaysia_zscore_60d", "TGT_Target_zscore_60d", "EWQ_France_zscore_60d", "IBEX_Spain_ret_20d", "PPL_PPL_ret_1d", "EWA_Australia_zscore_60d", "MO_AltriaMG_ret_1d", "US3Y_Rate_ret_5d", "ORCL_zscore_60d", "PG_ret_20d", "MSTR_Bitcoin3_ret_20d"], "is_new": true}, {"model_id": "new_h1_CALM_LightGBM_N20_t2", "algo": "LightGBM", "regime": "CALM", "horizon": 1, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "XLB_Materials_zscore_60d", "DHR_ret_1d", "EWY_Korea_zscore_60d", "BA_ret_1d", "heston_var_ev_h5", "SBUX_ret_5d", "MRK_Merck_zscore_60d", "US1Y_Rate_ret_20d", "CPB_CampbellSoup_vol_20d", "SPY_zscore_60d", "DAX_Germany_vol_20d", "HD_ret_20d", "CI_Cigna_vol_20d", "EMR_Emerson_ret_20d", "GE_ret_1d", "PAYX_Paychex_zscore_60d", "SO_SouthernCo_ret_5d", "DIS_vol_20d"], "is_new": true}, {"model_id": "new_h1_CALM_LightGBM_N20_t3", "algo": "LightGBM", "regime": "CALM", "horizon": 1, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "LMT_LockheedMartin_ret_1d", "AMZN_ret_5d", "NOC_Northrop_ret_20d", "BA_ret_1d", "MS_MorganStanley_zscore_60d", "SJM_JM_Smucker_ret_5d", "MRK_Merck_zscore_60d", "EWQ_France_ret_20d", "MSTR_Bitcoin3_ret_5d", "CCI_CrownCastle_vol_20d", "EWM_Malaysia_ret_1d", "AORD_AUS_zscore_60d", "INTC_ret_5d", "CPB_CampbellSoup_ret_5d", "hmm_p_stress", "IYR_US_REIT2_zscore_60d", "DIS_vol_20d", "DHR_ret_1d"], "is_new": true}, {"model_id": "new_h1_CALM_LightGBM_N20_t4", "algo": "LightGBM", "regime": "CALM", "horizon": 1, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "PPL_PPL_ret_1d", "PAYX_Paychex_vol_20d", "XLV_Health_zscore_60d", "3M_vol_20d", "CCI_CrownCastle_vol_20d", "AMGN_Amgen_ret_1d", "EXC_Exelon_ret_1d", "EWL_Switzerland_vol_20d", "DOW_Price_zscore_60d", "T_ret_1d", "XLF_Fin_vol_20d", "TED_Spread_zscore_60d", "NWL_Newell_ret_20d", "CTAS_Cintas_vol_20d", "HD_ret_5d", "EWQ_France_ret_20d", "HangSeng_HK_ret_5d", "DAX_Germany_vol_20d"], "is_new": true}, {"model_id": "new_h1_CALM_LightGBM_N20_t5", "algo": "LightGBM", "regime": "CALM", "horizon": 1, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "BTI_BritishAmerican_ret_20d", "CTAS_Cintas_vol_20d", "SJM_JM_Smucker_ret_1d", "VVIX_ret_20d", "TM_Telephone_ret_1d", "AMGN_Amgen_ret_1d", "gjr_condvar_h1", "SO_SouthernCo_ret_5d", "US5Y_Rate_ret_5d", "EWQ_France_zscore_60d", "DHR_vol_20d", "HD_zscore_60d", "Nikkei_Japan_vol_20d", "3M_ret_5d", "XLV_Health_zscore_60d", "LMT_LockheedMartin_vol_20d", "US3Y_Rate_ret_5d", "NOC_Northrop_ret_20d"], "is_new": true}, {"model_id": "new_h1_CALM_LightGBM_N20_t6", "algo": "LightGBM", "regime": "CALM", "horizon": 1, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "TED_Spread_vol_20d", "TXN_vol_20d", "SLB_Schlumberger_ret_1d", "US1Y_Rate_ret_20d", "AMT_AmericanTower_ret_1d", "MS_MorganStanley_ret_1d", "PPL_PPL_ret_1d", "AXP_Amex_ret_20d", "heston_ev_h3", "MRK_Merck_zscore_60d", "EWY_Korea_zscore_60d", "AMGN_Amgen_ret_1d", "LOW_Lowes_ret_20d", "BTI_BritishAmerican_ret_5d", "LLY_zscore_60d", "EWQ_France_zscore_60d", "spx_momentum_3d", "XOM_ret_20d"], "is_new": true}, {"model_id": "new_h1_CALM_LightGBM_N20_t7", "algo": "LightGBM", "regime": "CALM", "horizon": 1, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "LMT_LockheedMartin_vol_20d", "TXN_vol_20d", "Industrial_Production_zscore_60d", "heston_var_ev_h5", "EOG_EOGResources_vol_20d", "PG_ret_20d", "Retail_Sales_zscore_60d", "gjr_condvar_h1", "EQR_Equity_ret_1d", "DE_Deere_vol_20d", "BTI_BritishAmerican_ret_20d", "PFE_ret_1d", "ASX_Australia_vol_20d", "HUM_Humana_ret_5d", "EWG_Germany_ret_20d", "AXP_Amex_ret_20d", "AVB_AvalonBay_zscore_60d", "INTC_ret_1d"], "is_new": true}, {"model_id": "new_h1_CALM_LightGBM_N25_t0", "algo": "LightGBM", "regime": "CALM", "horizon": 1, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "ASX_Australia_vol_20d", "heston_var_ev_h7", "EWA_Australia_ret_1d", "HD_ret_5d", "LOW_Lowes_ret_20d", "AXP_Amex_vol_20d", "IYM_BasicMaterials_ret_20d", "GE_ret_1d", "DAX_Germany_vol_20d", "MO_AltriaMG_ret_1d", "ORCL_vol_20d", "M_Macys_vol_20d", "ORCL_zscore_60d", "MSTR_Bitcoin3_ret_5d", "Industrial_Production_zscore_60d", "ES_Evergy_ret_1d", "SJM_JM_Smucker_ret_1d", "XLF_Fin_vol_20d", "heston_ev_h3", "SPY_zscore_60d", "GD_GeneralDynamics_zscore_60d", "CI_Cigna_vol_20d", "NWL_Newell_ret_20d"], "is_new": true}, {"model_id": "new_h1_CALM_LightGBM_N25_t1", "algo": "LightGBM", "regime": "CALM", "horizon": 1, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "ITT_ITTInc_ret_5d", "CPB_CampbellSoup_vol_20d", "CCI_CrownCastle_vol_20d", "IWM_SmallCap_vol_20d", "HUM_Humana_ret_5d", "HangSeng_HK_vol_20d", "PAYX_Paychex_vol_20d", "XLB_Materials_zscore_60d", "Nikkei_Japan_vol_20d", "ENB_EnbridgeInc_ret_1d", "PAYX_Paychex_ret_20d", "LMT_LockheedMartin_ret_1d", "heston_var_ev_h7", "AVB_AvalonBay_zscore_60d", "LMT_LockheedMartin_vol_20d", "ORCL_vol_20d", "GILD_Gilead_ret_20d", "BDX_Becton_Dickinson_ret_20d", "Brent_Oil_FRED_ret_5d", "Michigan_Sentiment_ret_20d", "LUV_SouthwestAir_ret_5d", "SO_SouthernCo_ret_5d", "EFFR_vol_20d"], "is_new": true}, {"model_id": "new_h1_CALM_LightGBM_N25_t2", "algo": "LightGBM", "regime": "CALM", "horizon": 1, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "DAX_Germany_zscore_60d", "BTI_BritishAmerican_ret_5d", "EWQ_France_ret_20d", "heston_var_ev_h7", "LUV_SouthwestAir_ret_5d", "LOW_Lowes_ret_5d", "MSTR_Bitcoin3_ret_5d", "US3M_Rate_zscore_60d", "ASX_Australia_ret_5d", "XLB_Materials_zscore_60d", "EFFR_ret_1d", "PAYX_Paychex_zscore_60d", "MO_AltriaMG_ret_1d", "MRK_Merck_zscore_60d", "PAYX_Paychex_ret_20d", "Brent_Oil_FRED_ret_5d", "US3Y_Rate_ret_5d", "DHR_ret_1d", "TED_Spread_zscore_60d", "spx_abs_ret_max_5d", "MS_MorganStanley_zscore_60d", "INTC_ret_1d", "HD_ret_1d"], "is_new": true}, {"model_id": "new_h1_CALM_LightGBM_N25_t3", "algo": "LightGBM", "regime": "CALM", "horizon": 1, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "SJM_JM_Smucker_ret_1d", "SPY_zscore_60d", "MS_MorganStanley_ret_1d", "US30Y_Rate_ret_20d", "Core_PCE_zscore_60d", "FedFunds_zscore_60d", "EXC_Exelon_zscore_60d", "PAYX_Paychex_vol_20d", "EWJ_Japan_vol_20d", "XLK_Tech_zscore_60d", "HD_ret_5d", "SBUX_zscore_60d", "EWG_Germany_ret_20d", "DAX_Germany_vol_20d", "DE_Deere_vol_20d", "Brent_Oil_FRED_ret_20d", "PAYX_Paychex_zscore_60d", "EOG_EOGResources_vol_20d", "PAYX_Paychex_ret_20d", "CLX_Clorox_vol_20d", "INTC_ret_5d", "LLY_zscore_60d", "BTI_BritishAmerican_ret_20d"], "is_new": true}, {"model_id": "new_h1_CALM_LightGBM_N25_t4", "algo": "LightGBM", "regime": "CALM", "horizon": 1, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "heston_var_ev_h3", "EOG_EOGResources_vol_20d", "LOW_Lowes_ret_5d", "SLB_Schlumberger_ret_5d", "EWM_Malaysia_ret_1d", "DAX_Germany_vol_20d", "EWG_Germany_ret_20d", "DHR_ret_1d", "EWQ_France_ret_20d", "SCHW_Schwab_ret_5d", "AMZN_ret_5d", "spx_vol_5d", "US1Y_Rate_ret_20d", "US3Y_Rate_ret_5d", "MO_AltriaMG_ret_1d", "PLD_Prologis_ret_5d", "XOM_ret_1d", "DE_Deere_ret_5d", "CLX_Clorox_vol_20d", "BTI_BritishAmerican_ret_20d", "IBEX_Spain_ret_20d", "MS_MorganStanley_zscore_60d", "Retail_Sales_zscore_60d"], "is_new": true}, {"model_id": "new_h1_CALM_LightGBM_N25_t5", "algo": "LightGBM", "regime": "CALM", "horizon": 1, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "DAX_Germany_zscore_60d", "INTC_ret_5d", "heston_var_ev_h5", "Brent_Oil_FRED_ret_20d", "PAYX_Paychex_zscore_60d", "PLD_Prologis_ret_5d", "SPY_zscore_60d", "LLY_zscore_60d", "vix_mean_abs_ret_5d", "US1Y_Rate_ret_20d", "VOD_Vodafone_zscore_60d", "EQR_Equity_ret_1d", "DE_Deere_ret_5d", "Nikkei_Japan_zscore_60d", "SJM_JM_Smucker_ret_5d", "BDX_Becton_Dickinson_ret_20d", "DAX_Germany_vol_20d", "heston_var_ev_h7", "EFFR_ret_1d", "HangSeng_HK_ret_5d", "ENB_EnbridgeInc_ret_1d", "EMR_Emerson_ret_20d", "ITT_ITTInc_ret_5d"], "is_new": true}, {"model_id": "new_h1_CALM_LightGBM_N25_t6", "algo": "LightGBM", "regime": "CALM", "horizon": 1, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "CCI_CrownCastle_vol_20d", "EWY_Korea_zscore_60d", "EWC_Canada_zscore_60d", "heston_ev_h3", "US1Y_Rate_ret_20d", "GILD_Gilead_ret_20d", "DAX_Germany_zscore_60d", "XOM_ret_20d", "HangSeng_HK_ret_5d", "XLV_Health_zscore_60d", "TGT_Target_zscore_60d", "CMCSA_ret_1d", "SJM_JM_Smucker_ret_5d", "EOG_EOGResources_ret_5d", "VRP_ma5", "US6M_Rate_ret_20d", "EFFR_ret_1d", "XLY_Disc_vol_20d", "Industrial_Production_zscore_60d", "US3M_Rate_vol_20d", "AVB_AvalonBay_zscore_60d", "AORD_AUS_zscore_60d", "vix_mean_abs_ret_5d"], "is_new": true}, {"model_id": "new_h1_CALM_LightGBM_N25_t7", "algo": "LightGBM", "regime": "CALM", "horizon": 1, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "XLY_Disc_vol_20d", "heston_var_ev_h3", "AORD_AUS_zscore_60d", "SJM_JM_Smucker_ret_1d", "XOM_ret_20d", "TXN_vol_20d", "HD_zscore_60d", "heston_var_ev_h5", "HangSeng_HK_ret_5d", "SBUX_ret_5d", "EWS_Singapore_ret_5d", "Michigan_Sentiment_ret_20d", "HangSeng_HK_ret_1d", "PCAR_PaccarInc_ret_5d", "Brent_Oil_FRED_ret_20d", "BLK_BlackRock_zscore_60d", "EFFR_vol_20d", "ORCL_zscore_60d", "NOC_Northrop_ret_20d", "heston_ev_h3", "PG_ret_20d", "EWG_Germany_vol_20d", "JNJ_ret_1d"], "is_new": true}, {"model_id": "new_h1_CALM_LightGBM_N30_t0", "algo": "LightGBM", "regime": "CALM", "horizon": 1, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "LLY_zscore_60d", "M_Macys_vol_20d", "DE_Deere_ret_5d", "MO_AltriaMG_ret_1d", "SO_SouthernCo_ret_5d", "US30Y_Rate_ret_20d", "CPB_CampbellSoup_zscore_60d", "INTC_ret_5d", "EWQ_France_zscore_60d", "PLD_Prologis_ret_5d", "DAX_Germany_vol_20d", "GD_GeneralDynamics_zscore_60d", "PAYX_Paychex_zscore_60d", "EWY_Korea_ret_20d", "DHR_vol_20d", "CPB_CampbellSoup_ret_20d", "AXP_Amex_vol_20d", "HD_zscore_60d", "MSTR_Bitcoin3_ret_5d", "NVDA_vol_20d", "XLY_Disc_vol_20d", "EFFR_vol_20d", "IWM_SmallCap_vol_20d", "US5Y_Rate_ret_5d", "INTC_ret_1d", "EWA_Australia_ret_1d", "VOD_Vodafone_zscore_60d", "SJM_JM_Smucker_ret_1d"], "is_new": true}, {"model_id": "new_h1_CALM_LightGBM_N30_t1", "algo": "LightGBM", "regime": "CALM", "horizon": 1, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "SJM_JM_Smucker_ret_1d", "AMT_AmericanTower_ret_1d", "CMCSA_ret_1d", "MS_MorganStanley_ret_5d", "SLB_Schlumberger_ret_1d", "PFE_ret_1d", "EWQ_France_zscore_60d", "EWL_Switzerland_vol_20d", "US7Y_Rate_ret_20d", "LMT_LockheedMartin_vol_20d", "AXP_Amex_vol_20d", "gjr_condvar_h1", "HD_ret_20d", "SCHW_Schwab_ret_5d", "VVIX_ret_20d", "PPL_PPL_ret_1d", "HD_ret_5d", "TM_Telephone_ret_1d", "AMD_ret_1d", "CPB_CampbellSoup_zscore_60d", "Nikkei_Japan_zscore_60d", "DE_Deere_vol_20d", "FedFunds_zscore_60d", "LUV_SouthwestAir_ret_5d", "Michigan_Sentiment_ret_20d", "US6M_Rate_ret_20d", "AXP_Amex_ret_20d", "M_Macys_vol_20d"], "is_new": true}, {"model_id": "new_h1_CALM_LightGBM_N30_t2", "algo": "LightGBM", "regime": "CALM", "horizon": 1, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "Brent_Oil_FRED_ret_20d", "MSTR_Bitcoin3_ret_5d", "SLB_Schlumberger_ret_1d", "PAYX_Paychex_zscore_60d", "T_ret_1d", "HD_ret_1d", "XOM_ret_1d", "XLK_Tech_zscore_60d", "Core_PCE_zscore_60d", "DAX_Germany_zscore_60d", "CPB_CampbellSoup_ret_20d", "MS_MorganStanley_ret_5d", "MRK_Merck_zscore_60d", "Industrial_Production_zscore_60d", "US1Y_Rate_ret_5d", "LUV_SouthwestAir_ret_5d", "DHR_vol_20d", "EWC_Canada_zscore_60d", "SLB_Schlumberger_ret_5d", "M_Macys_vol_20d", "SJM_JM_Smucker_ret_5d", "AVB_AvalonBay_zscore_60d", "AXP_Amex_ret_20d", "ASX_Australia_vol_20d", "DHR_ret_1d", "IYR_US_REIT2_zscore_60d", "QQQ_vol_20d", "SPY_zscore_60d"], "is_new": true}, {"model_id": "new_h1_CALM_LightGBM_N30_t3", "algo": "LightGBM", "regime": "CALM", "horizon": 1, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "XLK_Tech_zscore_60d", "INTC_ret_5d", "EMR_Emerson_ret_20d", "MS_MorganStanley_ret_5d", "XLV_Health_zscore_60d", "HangSeng_HK_ret_1d", "VRP_ma5", "Industrial_Production_zscore_60d", "SLB_Schlumberger_ret_1d", "SLB_Schlumberger_ret_5d", "TGT_Target_zscore_60d", "SBUX_zscore_60d", "US3M_Rate_vol_20d", "hmm_p_stress", "QQQ_vol_20d", "AMGN_Amgen_ret_1d", "MO_AltriaMG_ret_1d", "IYM_BasicMaterials_ret_20d", "CTAS_Cintas_vol_20d", "NOC_Northrop_ret_20d", "US7Y_Rate_ret_20d", "HangSeng_HK_vol_20d", "AMZN_ret_5d", "spx_abs_ret_max_5d", "FedFunds_zscore_60d", "EWM_Malaysia_vol_20d", "BA_ret_1d", "Retail_Sales_zscore_60d"], "is_new": true}, {"model_id": "new_h1_CALM_LightGBM_N30_t4", "algo": "LightGBM", "regime": "CALM", "horizon": 1, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "ASX_Australia_ret_5d", "US3Y_Rate_ret_5d", "CTAS_Cintas_vol_20d", "EWA_Australia_ret_1d", "AMGN_Amgen_ret_1d", "ORCL_zscore_60d", "INTC_ret_1d", "NVDA_vol_20d", "DOW_Price_zscore_60d", "NFCI_ret_5d", "EWG_Germany_vol_20d", "heston_ev_h3", "SJM_JM_Smucker_ret_5d", "XLK_Tech_zscore_60d", "PPL_PPL_ret_1d", "BLK_BlackRock_zscore_60d", "DHR_ret_1d", "SPY_zscore_60d", "EWH_HongKong_ret_5d", "US30Y_Rate_ret_20d", "IWM_SmallCap_vol_20d", "Retail_Sales_zscore_60d", "TED_Spread_vol_20d", "MSTR_Bitcoin3_ret_1d", "heston_var_ev_h7", "EOG_EOGResources_vol_20d", "PFE_ret_1d", "TGT_Target_zscore_60d"], "is_new": true}, {"model_id": "new_h1_CALM_LightGBM_N30_t5", "algo": "LightGBM", "regime": "CALM", "horizon": 1, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWA_Australia_zscore_60d", "EWY_Korea_ret_20d", "HD_ret_1d", "US1Y_Rate_ret_5d", "HangSeng_HK_ret_1d", "EWM_Malaysia_ret_1d", "spx_abs_ret_max_5d", "HD_ret_5d", "EWH_HongKong_ret_5d", "SBUX_zscore_60d", "INTC_ret_5d", "Retail_Sales_zscore_60d", "LMT_LockheedMartin_ret_1d", "US30Y_Rate_ret_20d", "GILD_Gilead_ret_20d", "heston_var_ev_h5", "DAX_Germany_vol_20d", "US3Y_Rate_ret_5d", "TXN_vol_20d", "MSTR_Bitcoin3_ret_20d", "MS_MorganStanley_zscore_60d", "LLY_zscore_60d", "DIS_vol_20d", "HD_zscore_60d", "AVB_AvalonBay_zscore_60d", "XLK_Tech_zscore_60d", "CCI_CrownCastle_vol_20d", "VOD_Vodafone_zscore_60d"], "is_new": true}, {"model_id": "new_h1_CALM_LightGBM_N30_t6", "algo": "LightGBM", "regime": "CALM", "horizon": 1, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWG_Germany_ret_20d", "TM_Telephone_ret_1d", "GE_ret_1d", "US30Y_Rate_ret_20d", "spx_vol_5d", "DHR_ret_1d", "CPB_CampbellSoup_zscore_60d", "XLV_Health_zscore_60d", "VOD_Vodafone_zscore_60d", "EWQ_France_zscore_60d", "EWS_Singapore_ret_5d", "AXP_Amex_vol_20d", "TED_Spread_zscore_60d", "EWM_Malaysia_ret_1d", "EWM_Malaysia_zscore_60d", "BA_ret_1d", "CPB_CampbellSoup_ret_5d", "NFCI_ret_5d", "IYM_BasicMaterials_ret_20d", "PPL_PPL_ret_1d", "XLF_Fin_vol_20d", "hmm_p_stress", "3M_vol_20d", "T_ret_1d", "EFFR_vol_20d", "NEE_NextEra_ret_20d", "QQQ_vol_20d", "Retail_Sales_zscore_60d"], "is_new": true}, {"model_id": "new_h1_CALM_LightGBM_N30_t7", "algo": "LightGBM", "regime": "CALM", "horizon": 1, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "HD_ret_5d", "heston_var_ev_h7", "BLK_BlackRock_zscore_60d", "CLX_Clorox_vol_20d", "hmm_p_stress", "US3Y_Rate_ret_5d", "EWM_Malaysia_zscore_60d", "TGT_Target_zscore_60d", "ASX_Australia_vol_20d", "EOG_EOGResources_vol_20d", "NOC_Northrop_ret_20d", "EQIX_Equinix_ret_5d", "heston_var_ev_h3", "MS_MorganStanley_zscore_60d", "SCHW_Schwab_ret_5d", "ES_Evergy_ret_1d", "HD_zscore_60d", "XOM_ret_1d", "AMD_ret_1d", "HangSeng_HK_ret_1d", "US3M_Rate_zscore_60d", "CPB_CampbellSoup_zscore_60d", "IYR_US_REIT2_zscore_60d", "HD_ret_20d", "PCAR_PaccarInc_ret_5d", "TXN_vol_20d", "TED_Spread_vol_20d", "XLV_Health_zscore_60d"], "is_new": true}, {"model_id": "new_h1_CALM_GradientBoosting_N5_t0", "algo": "GradientBoosting", "regime": "CALM", "horizon": 1, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "CTAS_Cintas_vol_20d", "IYM_BasicMaterials_ret_20d", "EMR_Emerson_ret_20d"], "is_new": true}, {"model_id": "new_h1_CALM_GradientBoosting_N5_t1", "algo": "GradientBoosting", "regime": "CALM", "horizon": 1, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "AMD_ret_5d", "Industrial_Production_zscore_60d", "HUM_Humana_ret_5d"], "is_new": true}, {"model_id": "new_h1_CALM_GradientBoosting_N5_t2", "algo": "GradientBoosting", "regime": "CALM", "horizon": 1, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "Nikkei_Japan_vol_20d", "SO_SouthernCo_ret_5d", "IWM_SmallCap_vol_20d"], "is_new": true}, {"model_id": "new_h1_CALM_GradientBoosting_N5_t3", "algo": "GradientBoosting", "regime": "CALM", "horizon": 1, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "DHR_ret_1d", "Retail_Sales_zscore_60d", "ITT_ITTInc_ret_5d"], "is_new": true}, {"model_id": "new_h1_CALM_GradientBoosting_N5_t4", "algo": "GradientBoosting", "regime": "CALM", "horizon": 1, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWQ_France_ret_20d", "PAYX_Paychex_vol_20d", "GD_GeneralDynamics_zscore_60d"], "is_new": true}, {"model_id": "new_h1_CALM_GradientBoosting_N5_t5", "algo": "GradientBoosting", "regime": "CALM", "horizon": 1, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "spx_abs_ret_max_5d", "LOW_Lowes_ret_5d", "EWQ_France_ret_20d"], "is_new": true}, {"model_id": "new_h1_CALM_GradientBoosting_N5_t6", "algo": "GradientBoosting", "regime": "CALM", "horizon": 1, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EFFR_vol_20d", "Michigan_Sentiment_ret_20d", "SJM_JM_Smucker_ret_5d"], "is_new": true}, {"model_id": "new_h1_CALM_GradientBoosting_N5_t7", "algo": "GradientBoosting", "regime": "CALM", "horizon": 1, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "MO_AltriaMG_ret_1d", "XLV_Health_zscore_60d", "EWL_Switzerland_zscore_60d"], "is_new": true}, {"model_id": "new_h1_CALM_GradientBoosting_N8_t0", "algo": "GradientBoosting", "regime": "CALM", "horizon": 1, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "PCAR_PaccarInc_ret_5d", "XLF_Fin_vol_20d", "MRK_Merck_zscore_60d", "NOC_Northrop_ret_20d", "T_ret_1d", "PPL_PPL_ret_1d"], "is_new": true}, {"model_id": "new_h1_CALM_GradientBoosting_N8_t1", "algo": "GradientBoosting", "regime": "CALM", "horizon": 1, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "TED_Spread_vol_20d", "PAYX_Paychex_vol_20d", "TED_Spread_zscore_60d", "LOW_Lowes_ret_5d", "Brent_Oil_FRED_ret_5d", "US3M_Rate_vol_20d"], "is_new": true}, {"model_id": "new_h1_CALM_GradientBoosting_N8_t2", "algo": "GradientBoosting", "regime": "CALM", "horizon": 1, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "ENB_EnbridgeInc_ret_1d", "IWM_SmallCap_vol_20d", "T10Y2Y_Spread_ret_5d", "PAYX_Paychex_ret_20d", "Core_CPI_zscore_60d", "EWG_Germany_vol_20d"], "is_new": true}, {"model_id": "new_h1_CALM_GradientBoosting_N8_t3", "algo": "GradientBoosting", "regime": "CALM", "horizon": 1, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "BDX_Becton_Dickinson_ret_20d", "PCAR_PaccarInc_ret_5d", "DE_Deere_ret_5d", "SO_SouthernCo_ret_5d", "NOC_Northrop_ret_20d", "US3M_Rate_vol_20d"], "is_new": true}, {"model_id": "new_h1_CALM_GradientBoosting_N8_t4", "algo": "GradientBoosting", "regime": "CALM", "horizon": 1, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "MS_MorganStanley_zscore_60d", "EXC_Exelon_ret_1d", "3M_ret_5d", "EWL_Switzerland_vol_20d", "US6M_Rate_ret_20d", "Brent_Oil_FRED_ret_20d"], "is_new": true}, {"model_id": "new_h1_CALM_GradientBoosting_N8_t5", "algo": "GradientBoosting", "regime": "CALM", "horizon": 1, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "TM_Telephone_vol_20d", "BLK_BlackRock_zscore_60d", "BTI_BritishAmerican_ret_5d", "PPL_PPL_ret_1d", "ASX_Australia_ret_5d", "HD_ret_5d"], "is_new": true}, {"model_id": "new_h1_CALM_GradientBoosting_N8_t6", "algo": "GradientBoosting", "regime": "CALM", "horizon": 1, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "MRK_Merck_zscore_60d", "MSTR_Bitcoin3_ret_20d", "PAYX_Paychex_vol_20d", "MSTR_Bitcoin3_ret_1d", "CCI_CrownCastle_vol_20d", "spx_vol_5d"], "is_new": true}, {"model_id": "new_h1_CALM_GradientBoosting_N8_t7", "algo": "GradientBoosting", "regime": "CALM", "horizon": 1, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EQIX_Equinix_ret_5d", "TM_Telephone_ret_1d", "MS_MorganStanley_ret_5d", "TED_Spread_zscore_60d", "DAX_Germany_zscore_60d", "EWM_Malaysia_ret_1d"], "is_new": true}, {"model_id": "new_h1_CALM_GradientBoosting_N10_t0", "algo": "GradientBoosting", "regime": "CALM", "horizon": 1, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "TXN_vol_20d", "NWL_Newell_ret_20d", "TGT_Target_zscore_60d", "MSTR_Bitcoin3_ret_20d", "EWM_Malaysia_vol_20d", "EWY_Korea_ret_20d", "PFE_ret_1d", "LLY_zscore_60d"], "is_new": true}, {"model_id": "new_h1_CALM_GradientBoosting_N10_t1", "algo": "GradientBoosting", "regime": "CALM", "horizon": 1, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "ASX_Australia_ret_5d", "spx_vol_5d", "NVDA_vol_20d", "XLV_Health_zscore_60d", "AMGN_Amgen_ret_1d", "EWQ_France_zscore_60d", "EWG_Germany_vol_20d", "BTI_BritishAmerican_ret_5d"], "is_new": true}, {"model_id": "new_h1_CALM_GradientBoosting_N10_t2", "algo": "GradientBoosting", "regime": "CALM", "horizon": 1, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWS_Singapore_ret_5d", "AXP_Amex_ret_20d", "M_Macys_vol_20d", "ORCL_zscore_60d", "EWM_Malaysia_zscore_60d", "TM_Telephone_vol_20d", "EWL_Switzerland_vol_20d", "BDX_Becton_Dickinson_ret_20d"], "is_new": true}, {"model_id": "new_h1_CALM_GradientBoosting_N10_t3", "algo": "GradientBoosting", "regime": "CALM", "horizon": 1, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "US5Y_Rate_ret_5d", "WTI_Oil_FRED_zscore_60d", "ITT_ITTInc_ret_5d", "EWQ_France_ret_20d", "HangSeng_HK_ret_5d", "DHR_vol_20d", "spx_vol_5d", "T10Y2Y_Spread_ret_5d"], "is_new": true}, {"model_id": "new_h1_CALM_GradientBoosting_N10_t4", "algo": "GradientBoosting", "regime": "CALM", "horizon": 1, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "BLK_BlackRock_zscore_60d", "US1Y_Rate_ret_5d", "VRP_ma5", "EOG_EOGResources_vol_20d", "XOM_ret_20d", "Nikkei_Japan_vol_20d", "SCHW_Schwab_ret_5d", "AXP_Amex_vol_20d"], "is_new": true}, {"model_id": "new_h1_CALM_GradientBoosting_N10_t5", "algo": "GradientBoosting", "regime": "CALM", "horizon": 1, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "NEE_NextEra_ret_20d", "DOW_Price_zscore_60d", "HangSeng_HK_ret_1d", "MSTR_Bitcoin3_ret_1d", "ES_Evergy_ret_1d", "TM_Telephone_ret_1d", "QQQ_vol_20d", "EWQ_France_zscore_60d"], "is_new": true}, {"model_id": "new_h1_CALM_GradientBoosting_N10_t6", "algo": "GradientBoosting", "regime": "CALM", "horizon": 1, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "BDX_Becton_Dickinson_ret_20d", "US30Y_Rate_ret_20d", "NFCI_ret_5d", "NOC_Northrop_ret_20d", "US1Y_Rate_ret_5d", "TED_Spread_zscore_60d", "DE_Deere_vol_20d", "MS_MorganStanley_ret_1d"], "is_new": true}, {"model_id": "new_h1_CALM_GradientBoosting_N10_t7", "algo": "GradientBoosting", "regime": "CALM", "horizon": 1, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "heston_var_ev_h7", "CCI_CrownCastle_vol_20d", "PAYX_Paychex_zscore_60d", "MO_AltriaMG_ret_1d", "CMCSA_ret_1d", "TM_Telephone_ret_1d", "EWA_Australia_ret_1d", "BA_ret_1d"], "is_new": true}, {"model_id": "new_h1_CALM_GradientBoosting_N12_t0", "algo": "GradientBoosting", "regime": "CALM", "horizon": 1, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWA_Australia_ret_1d", "VOD_Vodafone_zscore_60d", "vix_mean_abs_ret_5d", "US30Y_Rate_ret_20d", "EWL_Switzerland_vol_20d", "Retail_Sales_zscore_60d", "EWH_HongKong_ret_5d", "IYR_US_REIT2_zscore_60d", "HangSeng_HK_ret_5d", "AMD_ret_5d"], "is_new": true}, {"model_id": "new_h1_CALM_GradientBoosting_N12_t1", "algo": "GradientBoosting", "regime": "CALM", "horizon": 1, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWS_Singapore_ret_5d", "XLB_Materials_zscore_60d", "EWG_Germany_vol_20d", "MS_MorganStanley_zscore_60d", "EWQ_France_zscore_60d", "US1Y_Rate_ret_20d", "ITT_ITTInc_ret_5d", "HD_ret_20d", "MSTR_Bitcoin3_ret_20d", "PLD_Prologis_ret_5d"], "is_new": true}, {"model_id": "new_h1_CALM_GradientBoosting_N12_t2", "algo": "GradientBoosting", "regime": "CALM", "horizon": 1, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "LOW_Lowes_ret_5d", "SO_SouthernCo_ret_5d", "SJM_JM_Smucker_ret_1d", "PLD_Prologis_ret_5d", "CPB_CampbellSoup_ret_5d", "PPL_PPL_ret_1d", "INTC_ret_5d", "EWG_Germany_vol_20d", "MO_AltriaMG_ret_1d", "INTC_ret_1d"], "is_new": true}, {"model_id": "new_h1_CALM_GradientBoosting_N12_t3", "algo": "GradientBoosting", "regime": "CALM", "horizon": 1, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "AMGN_Amgen_ret_1d", "EXC_Exelon_zscore_60d", "US1Y_Rate_ret_20d", "US7Y_Rate_ret_20d", "US6M_Rate_ret_20d", "NOC_Northrop_ret_20d", "HUM_Humana_ret_5d", "EFFR_ret_1d", "EWC_Canada_zscore_60d", "IYM_BasicMaterials_ret_20d"], "is_new": true}, {"model_id": "new_h1_CALM_GradientBoosting_N12_t4", "algo": "GradientBoosting", "regime": "CALM", "horizon": 1, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "T_ret_1d", "spx_abs_ret_max_5d", "Industrial_Production_zscore_60d", "EXC_Exelon_ret_1d", "SBUX_zscore_60d", "heston_var_ev_h5", "PFE_ret_1d", "EOG_EOGResources_ret_5d", "LMT_LockheedMartin_vol_20d", "DHR_ret_1d"], "is_new": true}, {"model_id": "new_h1_CALM_GradientBoosting_N12_t5", "algo": "GradientBoosting", "regime": "CALM", "horizon": 1, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "TED_Spread_vol_20d", "ASX_Australia_ret_5d", "Brent_Oil_FRED_ret_5d", "vix_mean_abs_ret_5d", "AXP_Amex_vol_20d", "3M_vol_20d", "NFCI_ret_5d", "DHR_ret_1d", "gjr_condvar_h1", "EWM_Malaysia_zscore_60d"], "is_new": true}, {"model_id": "new_h1_CALM_GradientBoosting_N12_t6", "algo": "GradientBoosting", "regime": "CALM", "horizon": 1, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "SLB_Schlumberger_ret_1d", "PCAR_PaccarInc_ret_5d", "MRK_Merck_zscore_60d", "BLK_BlackRock_zscore_60d", "XLY_Disc_vol_20d", "heston_var_ev_h3", "vix_acceleration_1d", "XLV_Health_zscore_60d", "SBUX_ret_5d", "spx_momentum_3d"], "is_new": true}, {"model_id": "new_h1_CALM_GradientBoosting_N12_t7", "algo": "GradientBoosting", "regime": "CALM", "horizon": 1, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "AMD_ret_5d", "AMGN_Amgen_ret_1d", "TED_Spread_zscore_60d", "GILD_Gilead_ret_20d", "TM_Telephone_ret_1d", "HD_ret_5d", "SJM_JM_Smucker_ret_5d", "Nikkei_Japan_zscore_60d", "EQIX_Equinix_ret_5d", "CPB_CampbellSoup_zscore_60d"], "is_new": true}, {"model_id": "new_h1_CALM_GradientBoosting_N15_t0", "algo": "GradientBoosting", "regime": "CALM", "horizon": 1, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "XLY_Disc_vol_20d", "TED_Spread_vol_20d", "EWY_Korea_zscore_60d", "LMT_LockheedMartin_vol_20d", "GD_GeneralDynamics_zscore_60d", "MS_MorganStanley_zscore_60d", "CCI_CrownCastle_vol_20d", "MRK_Merck_zscore_60d", "Michigan_Sentiment_ret_20d", "PFE_ret_1d", "US3M_Rate_vol_20d", "EWS_Singapore_ret_5d", "EWM_Malaysia_ret_1d"], "is_new": true}, {"model_id": "new_h1_CALM_GradientBoosting_N15_t1", "algo": "GradientBoosting", "regime": "CALM", "horizon": 1, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "Michigan_Sentiment_ret_20d", "MSTR_Bitcoin3_ret_20d", "MSTR_Bitcoin3_ret_1d", "US1Y_Rate_ret_20d", "QQQ_vol_20d", "BDX_Becton_Dickinson_ret_20d", "heston_var_ev_h3", "AMD_ret_1d", "EOG_EOGResources_ret_5d", "CI_Cigna_vol_20d", "SJM_JM_Smucker_ret_1d", "ORCL_vol_20d", "heston_var_ev_h7"], "is_new": true}, {"model_id": "new_h1_CALM_GradientBoosting_N15_t2", "algo": "GradientBoosting", "regime": "CALM", "horizon": 1, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWJ_Japan_vol_20d", "US3M_Rate_zscore_60d", "QQQ_vol_20d", "GE_ret_1d", "AXP_Amex_ret_20d", "Brent_Oil_FRED_ret_5d", "HD_ret_1d", "TXN_vol_20d", "US3M_Rate_vol_20d", "spx_momentum_3d", "ASX_Australia_ret_5d", "EQIX_Equinix_ret_5d", "INTC_ret_1d"], "is_new": true}, {"model_id": "new_h1_CALM_GradientBoosting_N15_t3", "algo": "GradientBoosting", "regime": "CALM", "horizon": 1, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "Brent_Oil_FRED_ret_5d", "HangSeng_HK_ret_1d", "EMR_Emerson_ret_20d", "US3M_Rate_vol_20d", "EWQ_France_zscore_60d", "US3M_Rate_zscore_60d", "MS_MorganStanley_ret_1d", "TM_Telephone_vol_20d", "IYM_BasicMaterials_ret_20d", "MSTR_Bitcoin3_ret_5d", "DHR_ret_1d", "CCI_CrownCastle_vol_20d", "SJM_JM_Smucker_ret_5d"], "is_new": true}, {"model_id": "new_h1_CALM_GradientBoosting_N15_t4", "algo": "GradientBoosting", "regime": "CALM", "horizon": 1, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "ORCL_vol_20d", "EWM_Malaysia_zscore_60d", "Brent_Oil_FRED_ret_5d", "JNJ_ret_1d", "XLB_Materials_zscore_60d", "XLV_Health_zscore_60d", "VVIX_ret_20d", "MS_MorganStanley_zscore_60d", "IWM_SmallCap_vol_20d", "AORD_AUS_zscore_60d", "heston_ev_h3", "GE_ret_1d", "PAYX_Paychex_zscore_60d"], "is_new": true}, {"model_id": "new_h1_CALM_GradientBoosting_N15_t5", "algo": "GradientBoosting", "regime": "CALM", "horizon": 1, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "Nikkei_Japan_zscore_60d", "ES_Evergy_ret_1d", "US7Y_Rate_ret_20d", "spx_vol_5d", "MSTR_Bitcoin3_ret_20d", "CI_Cigna_vol_20d", "XLY_Disc_vol_20d", "HUM_Humana_ret_5d", "EWY_Korea_ret_20d", "Brent_Oil_FRED_ret_20d", "GD_GeneralDynamics_zscore_60d", "EFFR_ret_1d", "heston_ev_h3"], "is_new": true}, {"model_id": "new_h1_CALM_GradientBoosting_N15_t6", "algo": "GradientBoosting", "regime": "CALM", "horizon": 1, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "CPB_CampbellSoup_zscore_60d", "CMCSA_ret_1d", "3M_vol_20d", "HangSeng_HK_vol_20d", "PAYX_Paychex_ret_20d", "LOW_Lowes_ret_5d", "GE_ret_1d", "CPB_CampbellSoup_vol_20d", "DE_Deere_ret_5d", "CLX_Clorox_vol_20d", "ES_Evergy_ret_1d", "Nikkei_Japan_zscore_60d", "PAYX_Paychex_vol_20d"], "is_new": true}, {"model_id": "new_h1_CALM_GradientBoosting_N15_t7", "algo": "GradientBoosting", "regime": "CALM", "horizon": 1, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "CLX_Clorox_vol_20d", "LMT_LockheedMartin_vol_20d", "US1Y_Rate_ret_20d", "CMCSA_ret_1d", "VVIX_ret_20d", "WTI_Oil_FRED_zscore_60d", "IWM_SmallCap_vol_20d", "MSTR_Bitcoin3_ret_5d", "AMD_ret_5d", "HD_zscore_60d", "NFCI_ret_5d", "MS_MorganStanley_ret_1d", "CI_Cigna_vol_20d"], "is_new": true}, {"model_id": "new_h1_CALM_GradientBoosting_N20_t0", "algo": "GradientBoosting", "regime": "CALM", "horizon": 1, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EOG_EOGResources_ret_5d", "DHR_ret_1d", "GD_GeneralDynamics_zscore_60d", "HangSeng_HK_ret_5d", "SCHW_Schwab_ret_5d", "DIS_vol_20d", "BTI_BritishAmerican_ret_20d", "ENB_EnbridgeInc_ret_1d", "ORCL_vol_20d", "GILD_Gilead_ret_20d", "NVDA_vol_20d", "HangSeng_HK_ret_1d", "SBUX_ret_5d", "Brent_Oil_FRED_ret_5d", "LOW_Lowes_ret_5d", "EWY_Korea_zscore_60d", "EWY_Korea_ret_20d", "XOM_ret_20d"], "is_new": true}, {"model_id": "new_h1_CALM_GradientBoosting_N20_t1", "algo": "GradientBoosting", "regime": "CALM", "horizon": 1, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "DE_Deere_vol_20d", "HangSeng_HK_ret_5d", "3M_vol_20d", "MS_MorganStanley_zscore_60d", "IYR_US_REIT2_zscore_60d", "XLY_Disc_vol_20d", "MSTR_Bitcoin3_ret_20d", "IYM_BasicMaterials_ret_20d", "ORCL_zscore_60d", "Nikkei_Japan_zscore_60d", "hmm_p_stress", "VVIX_ret_20d", "HangSeng_HK_ret_1d", "ITT_ITTInc_ret_5d", "XLB_Materials_zscore_60d", "EWG_Germany_ret_20d", "Retail_Sales_zscore_60d", "INTC_ret_1d"], "is_new": true}, {"model_id": "new_h1_CALM_GradientBoosting_N20_t2", "algo": "GradientBoosting", "regime": "CALM", "horizon": 1, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "heston_ev_h3", "SBUX_zscore_60d", "US5Y_Rate_ret_5d", "EWS_Singapore_ret_5d", "IWM_SmallCap_vol_20d", "MSTR_Bitcoin3_ret_1d", "HD_ret_1d", "MS_MorganStanley_ret_5d", "LUV_SouthwestAir_ret_5d", "Core_CPI_zscore_60d", "US3M_Rate_vol_20d", "EWG_Germany_ret_20d", "VRP_ma5", "TXN_vol_20d", "EQR_Equity_ret_1d", "T_ret_1d", "AMZN_ret_5d", "PG_ret_20d"], "is_new": true}, {"model_id": "new_h1_CALM_GradientBoosting_N20_t3", "algo": "GradientBoosting", "regime": "CALM", "horizon": 1, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "BTI_BritishAmerican_ret_5d", "NEE_NextEra_ret_20d", "SJM_JM_Smucker_ret_1d", "DHR_ret_1d", "heston_ev_h3", "HangSeng_HK_ret_5d", "EWQ_France_zscore_60d", "EWY_Korea_zscore_60d", "spx_momentum_3d", "EFFR_ret_1d", "NWL_Newell_ret_20d", "MSTR_Bitcoin3_ret_1d", "Brent_Oil_FRED_ret_20d", "PAYX_Paychex_vol_20d", "WTI_Oil_FRED_zscore_60d", "Retail_Sales_zscore_60d", "US7Y_Rate_ret_20d", "M_Macys_vol_20d"], "is_new": true}, {"model_id": "new_h1_CALM_GradientBoosting_N20_t4", "algo": "GradientBoosting", "regime": "CALM", "horizon": 1, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "TM_Telephone_ret_1d", "VVIX_ret_20d", "heston_var_ev_h3", "CI_Cigna_vol_20d", "CPB_CampbellSoup_zscore_60d", "EWY_Korea_ret_20d", "spx_momentum_3d", "vix_acceleration_1d", "LMT_LockheedMartin_vol_20d", "ENB_EnbridgeInc_ret_1d", "heston_var_ev_h7", "XLK_Tech_zscore_60d", "AXP_Amex_ret_20d", "US3M_Rate_vol_20d", "M_Macys_vol_20d", "IYM_BasicMaterials_ret_20d", "Nikkei_Japan_zscore_60d", "AMD_ret_1d"], "is_new": true}, {"model_id": "new_h1_CALM_GradientBoosting_N20_t5", "algo": "GradientBoosting", "regime": "CALM", "horizon": 1, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "PAYX_Paychex_vol_20d", "MRK_Merck_zscore_60d", "MS_MorganStanley_ret_5d", "PG_ret_20d", "VOD_Vodafone_zscore_60d", "SJM_JM_Smucker_ret_5d", "DOW_Price_zscore_60d", "LLY_zscore_60d", "AMZN_ret_5d", "SBUX_ret_5d", "CPB_CampbellSoup_vol_20d", "HUM_Humana_ret_5d", "NOC_Northrop_ret_20d", "CI_Cigna_vol_20d", "Core_CPI_zscore_60d", "DHR_vol_20d", "LMT_LockheedMartin_ret_1d", "IBEX_Spain_ret_20d"], "is_new": true}, {"model_id": "new_h1_CALM_GradientBoosting_N20_t6", "algo": "GradientBoosting", "regime": "CALM", "horizon": 1, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "LMT_LockheedMartin_ret_1d", "QQQ_vol_20d", "CPB_CampbellSoup_ret_20d", "ORCL_zscore_60d", "PAYX_Paychex_zscore_60d", "INTC_ret_5d", "BTI_BritishAmerican_ret_5d", "HangSeng_HK_ret_1d", "AXP_Amex_ret_20d", "Core_CPI_zscore_60d", "AMT_AmericanTower_ret_1d", "JNJ_ret_1d", "heston_var_ev_h7", "CPB_CampbellSoup_zscore_60d", "ENB_EnbridgeInc_ret_1d", "EWS_Singapore_ret_5d", "SPY_zscore_60d", "EOG_EOGResources_vol_20d"], "is_new": true}, {"model_id": "new_h1_CALM_GradientBoosting_N20_t7", "algo": "GradientBoosting", "regime": "CALM", "horizon": 1, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "PG_ret_20d", "EWM_Malaysia_zscore_60d", "Michigan_Sentiment_ret_20d", "gjr_condvar_h1", "ES_Evergy_ret_1d", "heston_var_ev_h3", "spx_momentum_3d", "Nikkei_Japan_zscore_60d", "MS_MorganStanley_zscore_60d", "AORD_AUS_zscore_60d", "XLV_Health_zscore_60d", "EWC_Canada_zscore_60d", "PAYX_Paychex_zscore_60d", "VRP_ma5", "Nikkei_Japan_vol_20d", "MS_MorganStanley_ret_1d", "3M_vol_20d", "EWJ_Japan_vol_20d"], "is_new": true}, {"model_id": "new_h1_CALM_GradientBoosting_N25_t0", "algo": "GradientBoosting", "regime": "CALM", "horizon": 1, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "TM_Telephone_ret_1d", "ES_Evergy_ret_1d", "LUV_SouthwestAir_ret_5d", "IWM_SmallCap_vol_20d", "EWC_Canada_zscore_60d", "PFE_ret_1d", "Core_CPI_zscore_60d", "FedFunds_zscore_60d", "SBUX_zscore_60d", "TED_Spread_zscore_60d", "INTC_ret_1d", "VVIX_ret_20d", "MS_MorganStanley_ret_1d", "XLK_Tech_zscore_60d", "MS_MorganStanley_ret_5d", "VOD_Vodafone_zscore_60d", "heston_var_ev_h7", "LMT_LockheedMartin_ret_1d", "LMT_LockheedMartin_vol_20d", "hmm_p_stress", "EOG_EOGResources_vol_20d", "HangSeng_HK_ret_1d", "CI_Cigna_vol_20d"], "is_new": true}, {"model_id": "new_h1_CALM_GradientBoosting_N25_t1", "algo": "GradientBoosting", "regime": "CALM", "horizon": 1, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWQ_France_zscore_60d", "HangSeng_HK_vol_20d", "Core_CPI_zscore_60d", "EQR_Equity_ret_1d", "MRK_Merck_zscore_60d", "DHR_vol_20d", "SLB_Schlumberger_ret_5d", "MS_MorganStanley_ret_1d", "NVDA_vol_20d", "VVIX_ret_20d", "DHR_ret_1d", "US5Y_Rate_ret_5d", "MSTR_Bitcoin3_ret_5d", "NWL_Newell_ret_20d", "ES_Evergy_ret_1d", "SBUX_vol_20d", "TM_Telephone_ret_1d", "EOG_EOGResources_vol_20d", "US1Y_Rate_ret_5d", "ENB_EnbridgeInc_ret_1d", "SBUX_zscore_60d", "HD_ret_5d", "gjr_condvar_h1"], "is_new": true}, {"model_id": "new_h1_CALM_GradientBoosting_N25_t2", "algo": "GradientBoosting", "regime": "CALM", "horizon": 1, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "LOW_Lowes_ret_20d", "AMD_ret_1d", "SBUX_vol_20d", "DE_Deere_vol_20d", "EWA_Australia_zscore_60d", "MSTR_Bitcoin3_ret_5d", "US30Y_Rate_ret_20d", "SLB_Schlumberger_ret_1d", "SCHW_Schwab_ret_5d", "NEE_NextEra_ret_20d", "SBUX_zscore_60d", "NOC_Northrop_ret_20d", "PAYX_Paychex_ret_20d", "gjr_condvar_h1", "ENB_EnbridgeInc_ret_1d", "CI_Cigna_vol_20d", "US3M_Rate_vol_20d", "AORD_AUS_zscore_60d", "EWL_Switzerland_zscore_60d", "HD_zscore_60d", "ASX_Australia_ret_5d", "GILD_Gilead_ret_20d", "EWM_Malaysia_ret_1d"], "is_new": true}, {"model_id": "new_h1_CALM_GradientBoosting_N25_t3", "algo": "GradientBoosting", "regime": "CALM", "horizon": 1, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "US3M_Rate_vol_20d", "SCHW_Schwab_ret_5d", "T_ret_1d", "SLB_Schlumberger_ret_5d", "CTAS_Cintas_vol_20d", "US3Y_Rate_ret_5d", "Core_CPI_zscore_60d", "GD_GeneralDynamics_zscore_60d", "TXN_vol_20d", "LUV_SouthwestAir_ret_5d", "TM_Telephone_vol_20d", "AVB_AvalonBay_zscore_60d", "ORCL_zscore_60d", "heston_var_ev_h3", "EWL_Switzerland_vol_20d", "NWL_Newell_ret_20d", "CPB_CampbellSoup_vol_20d", "EWY_Korea_ret_20d", "HangSeng_HK_ret_1d", "ASX_Australia_ret_5d", "PAYX_Paychex_zscore_60d", "FedFunds_zscore_60d", "LMT_LockheedMartin_ret_1d"], "is_new": true}, {"model_id": "new_h1_CALM_GradientBoosting_N25_t4", "algo": "GradientBoosting", "regime": "CALM", "horizon": 1, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWH_HongKong_ret_5d", "VOD_Vodafone_zscore_60d", "LMT_LockheedMartin_ret_1d", "XLK_Tech_zscore_60d", "gjr_condvar_h1", "Nikkei_Japan_vol_20d", "ENB_EnbridgeInc_ret_1d", "XLV_Health_zscore_60d", "DHR_ret_1d", "EWG_Germany_vol_20d", "heston_ev_h3", "BLK_BlackRock_zscore_60d", "US3M_Rate_zscore_60d", "spx_vol_5d", "TGT_Target_zscore_60d", "HD_ret_1d", "VVIX_ret_20d", "TM_Telephone_ret_1d", "EOG_EOGResources_ret_5d", "ITT_ITTInc_ret_5d", "HD_zscore_60d", "VRP_ma5", "US3Y_Rate_ret_5d"], "is_new": true}, {"model_id": "new_h1_CALM_GradientBoosting_N25_t5", "algo": "GradientBoosting", "regime": "CALM", "horizon": 1, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "BLK_BlackRock_zscore_60d", "GILD_Gilead_ret_20d", "spx_abs_ret_max_5d", "DAX_Germany_zscore_60d", "DE_Deere_ret_5d", "EWY_Korea_ret_20d", "DIS_vol_20d", "ITT_ITTInc_ret_5d", "hmm_p_stress", "MS_MorganStanley_ret_5d", "SBUX_zscore_60d", "DHR_vol_20d", "US5Y_Rate_ret_5d", "GD_GeneralDynamics_zscore_60d", "NOC_Northrop_ret_20d", "MSTR_Bitcoin3_ret_1d", "PAYX_Paychex_zscore_60d", "Brent_Oil_FRED_ret_5d", "CMCSA_ret_1d", "Michigan_Sentiment_ret_20d", "SO_SouthernCo_ret_5d", "HD_ret_1d", "MO_AltriaMG_ret_1d"], "is_new": true}, {"model_id": "new_h1_CALM_GradientBoosting_N25_t6", "algo": "GradientBoosting", "regime": "CALM", "horizon": 1, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "CMCSA_ret_1d", "PFE_ret_1d", "heston_ev_h3", "LOW_Lowes_ret_20d", "BTI_BritishAmerican_ret_20d", "TGT_Target_zscore_60d", "MS_MorganStanley_zscore_60d", "EWL_Switzerland_vol_20d", "Nikkei_Japan_vol_20d", "EWJ_Japan_vol_20d", "SCHW_Schwab_ret_5d", "GD_GeneralDynamics_zscore_60d", "ITT_ITTInc_ret_5d", "HD_ret_20d", "Industrial_Production_zscore_60d", "T_ret_1d", "US5Y_Rate_ret_5d", "DE_Deere_ret_5d", "US3M_Rate_vol_20d", "SBUX_vol_20d", "JNJ_ret_1d", "AXP_Amex_vol_20d", "AXP_Amex_ret_20d"], "is_new": true}, {"model_id": "new_h1_CALM_GradientBoosting_N25_t7", "algo": "GradientBoosting", "regime": "CALM", "horizon": 1, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EFFR_vol_20d", "ORCL_zscore_60d", "PCAR_PaccarInc_ret_5d", "3M_ret_5d", "Nikkei_Japan_vol_20d", "EWG_Germany_ret_20d", "DE_Deere_ret_5d", "HangSeng_HK_ret_5d", "vix_mean_abs_ret_5d", "GILD_Gilead_ret_20d", "CMCSA_ret_1d", "LOW_Lowes_ret_5d", "EWY_Korea_zscore_60d", "CI_Cigna_vol_20d", "MRK_Merck_zscore_60d", "ENB_EnbridgeInc_ret_1d", "BA_ret_1d", "MS_MorganStanley_zscore_60d", "HangSeng_HK_vol_20d", "EWM_Malaysia_zscore_60d", "spx_abs_ret_max_5d", "gjr_condvar_h1", "CPB_CampbellSoup_ret_20d"], "is_new": true}, {"model_id": "new_h1_CALM_GradientBoosting_N30_t0", "algo": "GradientBoosting", "regime": "CALM", "horizon": 1, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "DE_Deere_ret_5d", "VRP_ma5", "EWM_Malaysia_ret_1d", "SPY_zscore_60d", "PAYX_Paychex_ret_20d", "AMD_ret_1d", "hmm_p_stress", "JNJ_ret_1d", "Core_PCE_zscore_60d", "NWL_Newell_ret_20d", "vix_mean_abs_ret_5d", "Retail_Sales_zscore_60d", "Brent_Oil_FRED_ret_20d", "EWM_Malaysia_vol_20d", "MSTR_Bitcoin3_ret_5d", "CLX_Clorox_vol_20d", "CPB_CampbellSoup_vol_20d", "XLF_Fin_vol_20d", "INTC_ret_1d", "MO_AltriaMG_ret_1d", "HangSeng_HK_ret_1d", "DHR_ret_1d", "MSTR_Bitcoin3_ret_20d", "DE_Deere_vol_20d", "DHR_vol_20d", "BTI_BritishAmerican_ret_5d", "TED_Spread_zscore_60d", "US3M_Rate_vol_20d"], "is_new": true}, {"model_id": "new_h1_CALM_GradientBoosting_N30_t1", "algo": "GradientBoosting", "regime": "CALM", "horizon": 1, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWJ_Japan_vol_20d", "XOM_ret_20d", "DOW_Price_zscore_60d", "NFCI_ret_5d", "XLV_Health_zscore_60d", "PAYX_Paychex_ret_20d", "EWL_Switzerland_vol_20d", "CLX_Clorox_vol_20d", "CPB_CampbellSoup_ret_5d", "DHR_ret_1d", "DAX_Germany_vol_20d", "VOD_Vodafone_zscore_60d", "MSTR_Bitcoin3_ret_5d", "LMT_LockheedMartin_ret_1d", "Nikkei_Japan_vol_20d", "INTC_ret_1d", "heston_ev_h3", "EWY_Korea_ret_20d", "GILD_Gilead_ret_20d", "CI_Cigna_vol_20d", "XLB_Materials_zscore_60d", "US1Y_Rate_ret_20d", "HD_ret_20d", "AMZN_ret_5d", "HangSeng_HK_ret_5d", "EWG_Germany_vol_20d", "SPY_zscore_60d", "SLB_Schlumberger_ret_5d"], "is_new": true}, {"model_id": "new_h1_CALM_GradientBoosting_N30_t2", "algo": "GradientBoosting", "regime": "CALM", "horizon": 1, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWQ_France_zscore_60d", "SCHW_Schwab_ret_5d", "MSTR_Bitcoin3_ret_5d", "CTAS_Cintas_vol_20d", "vix_mean_abs_ret_5d", "SJM_JM_Smucker_ret_1d", "heston_var_ev_h3", "VVIX_ret_20d", "Brent_Oil_FRED_ret_20d", "TED_Spread_zscore_60d", "AMD_ret_5d", "LOW_Lowes_ret_5d", "INTC_ret_5d", "DE_Deere_ret_5d", "MS_MorganStanley_ret_1d", "US3M_Rate_zscore_60d", "NVDA_vol_20d", "LUV_SouthwestAir_ret_5d", "TXN_vol_20d", "EOG_EOGResources_vol_20d", "PPL_PPL_ret_1d", "heston_var_ev_h7", "AMGN_Amgen_ret_1d", "BTI_BritishAmerican_ret_5d", "gjr_condvar_h1", "FedFunds_zscore_60d", "EWL_Switzerland_vol_20d", "AVB_AvalonBay_zscore_60d"], "is_new": true}, {"model_id": "new_h1_CALM_GradientBoosting_N30_t3", "algo": "GradientBoosting", "regime": "CALM", "horizon": 1, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "NWL_Newell_ret_20d", "M_Macys_vol_20d", "CI_Cigna_vol_20d", "heston_var_ev_h3", "TM_Telephone_ret_1d", "spx_abs_ret_max_5d", "XLV_Health_zscore_60d", "EFFR_ret_1d", "HD_ret_1d", "spx_vol_5d", "IBEX_Spain_ret_20d", "CCI_CrownCastle_vol_20d", "SLB_Schlumberger_ret_5d", "EWG_Germany_vol_20d", "heston_var_ev_h5", "hmm_p_stress", "IWM_SmallCap_vol_20d", "MS_MorganStanley_ret_5d", "VOD_Vodafone_zscore_60d", "BTI_BritishAmerican_ret_20d", "gjr_condvar_h1", "ORCL_vol_20d", "EWQ_France_zscore_60d", "BDX_Becton_Dickinson_ret_20d", "EWS_Singapore_ret_5d", "CLX_Clorox_vol_20d", "WTI_Oil_FRED_zscore_60d", "Core_CPI_zscore_60d"], "is_new": true}, {"model_id": "new_h1_CALM_GradientBoosting_N30_t4", "algo": "GradientBoosting", "regime": "CALM", "horizon": 1, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "Core_PCE_zscore_60d", "AMD_ret_5d", "DHR_ret_1d", "AMD_ret_1d", "EMR_Emerson_ret_20d", "SJM_JM_Smucker_ret_1d", "LLY_zscore_60d", "T10Y2Y_Spread_ret_5d", "SPY_zscore_60d", "EXC_Exelon_zscore_60d", "ENB_EnbridgeInc_ret_1d", "MRK_Merck_zscore_60d", "DAX_Germany_zscore_60d", "GD_GeneralDynamics_zscore_60d", "vix_mean_abs_ret_5d", "AORD_AUS_zscore_60d", "IYR_US_REIT2_zscore_60d", "Nikkei_Japan_vol_20d", "GE_ret_1d", "TED_Spread_vol_20d", "BTI_BritishAmerican_ret_20d", "FedFunds_zscore_60d", "US5Y_Rate_ret_5d", "Brent_Oil_FRED_ret_5d", "hmm_p_stress", "PCAR_PaccarInc_ret_5d", "SLB_Schlumberger_ret_5d", "LOW_Lowes_ret_5d"], "is_new": true}, {"model_id": "new_h1_CALM_GradientBoosting_N30_t5", "algo": "GradientBoosting", "regime": "CALM", "horizon": 1, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EFFR_vol_20d", "EXC_Exelon_ret_1d", "CPB_CampbellSoup_vol_20d", "US3Y_Rate_ret_5d", "Core_CPI_zscore_60d", "EWM_Malaysia_vol_20d", "SLB_Schlumberger_ret_5d", "3M_ret_5d", "AXP_Amex_ret_20d", "AMD_ret_5d", "EWS_Singapore_ret_5d", "heston_var_ev_h3", "PPL_PPL_ret_1d", "LUV_SouthwestAir_ret_5d", "MSTR_Bitcoin3_ret_5d", "DOW_Price_zscore_60d", "EQIX_Equinix_ret_5d", "ASX_Australia_vol_20d", "US1Y_Rate_ret_20d", "NVDA_vol_20d", "EWM_Malaysia_ret_1d", "ITT_ITTInc_ret_5d", "EWY_Korea_ret_20d", "NOC_Northrop_ret_20d", "INTC_ret_5d", "GE_ret_1d", "Brent_Oil_FRED_ret_5d", "MRK_Merck_zscore_60d"], "is_new": true}, {"model_id": "new_h1_CALM_GradientBoosting_N30_t6", "algo": "GradientBoosting", "regime": "CALM", "horizon": 1, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "SPY_zscore_60d", "AXP_Amex_ret_20d", "M_Macys_vol_20d", "BTI_BritishAmerican_ret_5d", "PLD_Prologis_ret_5d", "HD_ret_20d", "IBEX_Spain_ret_20d", "IWM_SmallCap_vol_20d", "CLX_Clorox_vol_20d", "US1Y_Rate_ret_20d", "PCAR_PaccarInc_ret_5d", "AMD_ret_1d", "MO_AltriaMG_ret_1d", "EWG_Germany_ret_20d", "HD_zscore_60d", "US3M_Rate_vol_20d", "HangSeng_HK_ret_5d", "EWS_Singapore_ret_5d", "PG_ret_20d", "XOM_ret_1d", "VRP_ma5", "SCHW_Schwab_ret_5d", "EWC_Canada_zscore_60d", "TXN_vol_20d", "XLB_Materials_zscore_60d", "DOW_Price_zscore_60d", "3M_vol_20d", "NEE_NextEra_ret_20d"], "is_new": true}, {"model_id": "new_h1_CALM_GradientBoosting_N30_t7", "algo": "GradientBoosting", "regime": "CALM", "horizon": 1, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "IYR_US_REIT2_zscore_60d", "3M_ret_5d", "spx_momentum_3d", "MSTR_Bitcoin3_ret_1d", "HangSeng_HK_vol_20d", "CPB_CampbellSoup_vol_20d", "EWG_Germany_vol_20d", "TED_Spread_zscore_60d", "EWQ_France_ret_20d", "NVDA_vol_20d", "SBUX_zscore_60d", "BLK_BlackRock_zscore_60d", "DHR_vol_20d", "HD_ret_1d", "CMCSA_ret_1d", "AXP_Amex_ret_20d", "EWM_Malaysia_zscore_60d", "T10Y2Y_Spread_ret_5d", "NFCI_ret_5d", "MRK_Merck_zscore_60d", "PFE_ret_1d", "CI_Cigna_vol_20d", "US6M_Rate_ret_20d", "CPB_CampbellSoup_ret_20d", "SLB_Schlumberger_ret_1d", "DE_Deere_vol_20d", "MS_MorganStanley_ret_5d", "Brent_Oil_FRED_ret_5d"], "is_new": true}, {"model_id": "new_h1_CALM_RandomForest_N5_t0", "algo": "RandomForest", "regime": "CALM", "horizon": 1, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWQ_France_ret_20d", "HD_zscore_60d", "EXC_Exelon_zscore_60d"], "is_new": true}, {"model_id": "new_h1_CALM_RandomForest_N5_t1", "algo": "RandomForest", "regime": "CALM", "horizon": 1, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "PAYX_Paychex_zscore_60d", "GE_ret_1d", "EWM_Malaysia_zscore_60d"], "is_new": true}, {"model_id": "new_h1_CALM_RandomForest_N5_t2", "algo": "RandomForest", "regime": "CALM", "horizon": 1, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EFFR_ret_1d", "CPB_CampbellSoup_zscore_60d", "ENB_EnbridgeInc_ret_1d"], "is_new": true}, {"model_id": "new_h1_CALM_RandomForest_N5_t3", "algo": "RandomForest", "regime": "CALM", "horizon": 1, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "Core_PCE_zscore_60d", "TM_Telephone_ret_1d", "LOW_Lowes_ret_20d"], "is_new": true}, {"model_id": "new_h1_CALM_RandomForest_N5_t4", "algo": "RandomForest", "regime": "CALM", "horizon": 1, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "FedFunds_zscore_60d", "Core_PCE_zscore_60d", "CMCSA_ret_1d"], "is_new": true}, {"model_id": "new_h1_CALM_RandomForest_N5_t5", "algo": "RandomForest", "regime": "CALM", "horizon": 1, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "IWM_SmallCap_vol_20d", "DIS_vol_20d", "CMCSA_ret_1d"], "is_new": true}, {"model_id": "new_h1_CALM_RandomForest_N5_t6", "algo": "RandomForest", "regime": "CALM", "horizon": 1, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "SO_SouthernCo_ret_5d", "AXP_Amex_vol_20d", "HangSeng_HK_ret_1d"], "is_new": true}, {"model_id": "new_h1_CALM_RandomForest_N5_t7", "algo": "RandomForest", "regime": "CALM", "horizon": 1, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "PAYX_Paychex_zscore_60d", "US1Y_Rate_ret_5d", "T_ret_1d"], "is_new": true}, {"model_id": "new_h1_CALM_RandomForest_N8_t0", "algo": "RandomForest", "regime": "CALM", "horizon": 1, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "DHR_ret_1d", "DAX_Germany_vol_20d", "AMD_ret_5d", "BLK_BlackRock_zscore_60d", "EWL_Switzerland_zscore_60d", "DE_Deere_ret_5d"], "is_new": true}, {"model_id": "new_h1_CALM_RandomForest_N8_t1", "algo": "RandomForest", "regime": "CALM", "horizon": 1, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "US7Y_Rate_ret_20d", "VOD_Vodafone_zscore_60d", "EXC_Exelon_zscore_60d", "Brent_Oil_FRED_ret_5d", "AMT_AmericanTower_ret_1d", "BTI_BritishAmerican_ret_20d"], "is_new": true}, {"model_id": "new_h1_CALM_RandomForest_N8_t2", "algo": "RandomForest", "regime": "CALM", "horizon": 1, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "IYM_BasicMaterials_ret_20d", "DAX_Germany_vol_20d", "AVB_AvalonBay_zscore_60d", "PG_ret_20d", "AXP_Amex_vol_20d", "EOG_EOGResources_ret_5d"], "is_new": true}, {"model_id": "new_h1_CALM_RandomForest_N8_t3", "algo": "RandomForest", "regime": "CALM", "horizon": 1, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "AORD_AUS_zscore_60d", "MSTR_Bitcoin3_ret_20d", "Nikkei_Japan_vol_20d", "EOG_EOGResources_vol_20d", "LMT_LockheedMartin_vol_20d", "EWJ_Japan_vol_20d"], "is_new": true}, {"model_id": "new_h1_CALM_RandomForest_N8_t4", "algo": "RandomForest", "regime": "CALM", "horizon": 1, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "DOW_Price_zscore_60d", "ASX_Australia_ret_5d", "XLV_Health_zscore_60d", "XLY_Disc_vol_20d", "EWS_Singapore_ret_5d", "heston_var_ev_h7"], "is_new": true}, {"model_id": "new_h1_CALM_RandomForest_N8_t5", "algo": "RandomForest", "regime": "CALM", "horizon": 1, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "BLK_BlackRock_zscore_60d", "CLX_Clorox_vol_20d", "HD_ret_1d", "MSTR_Bitcoin3_ret_5d", "DHR_ret_1d", "NVDA_vol_20d"], "is_new": true}, {"model_id": "new_h1_CALM_RandomForest_N8_t6", "algo": "RandomForest", "regime": "CALM", "horizon": 1, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "LOW_Lowes_ret_20d", "Retail_Sales_zscore_60d", "TXN_vol_20d", "SBUX_vol_20d", "LLY_zscore_60d", "US3M_Rate_vol_20d"], "is_new": true}, {"model_id": "new_h1_CALM_RandomForest_N8_t7", "algo": "RandomForest", "regime": "CALM", "horizon": 1, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EQR_Equity_ret_1d", "AMZN_ret_5d", "INTC_ret_5d", "VVIX_ret_20d", "EWL_Switzerland_zscore_60d", "AORD_AUS_zscore_60d"], "is_new": true}, {"model_id": "new_h1_CALM_RandomForest_N10_t0", "algo": "RandomForest", "regime": "CALM", "horizon": 1, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "US7Y_Rate_ret_20d", "EWM_Malaysia_zscore_60d", "EFFR_vol_20d", "CPB_CampbellSoup_ret_5d", "EWY_Korea_ret_20d", "spx_momentum_3d", "EWQ_France_zscore_60d", "GD_GeneralDynamics_zscore_60d"], "is_new": true}, {"model_id": "new_h1_CALM_RandomForest_N10_t1", "algo": "RandomForest", "regime": "CALM", "horizon": 1, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "LMT_LockheedMartin_vol_20d", "EOG_EOGResources_vol_20d", "EQIX_Equinix_ret_5d", "EWQ_France_ret_20d", "VRP_ma5", "INTC_ret_5d", "LOW_Lowes_ret_20d", "AORD_AUS_zscore_60d"], "is_new": true}, {"model_id": "new_h1_CALM_RandomForest_N10_t2", "algo": "RandomForest", "regime": "CALM", "horizon": 1, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "SCHW_Schwab_ret_5d", "TM_Telephone_vol_20d", "IBEX_Spain_ret_20d", "EOG_EOGResources_vol_20d", "SBUX_zscore_60d", "ASX_Australia_vol_20d", "CMCSA_ret_1d", "BDX_Becton_Dickinson_ret_20d"], "is_new": true}, {"model_id": "new_h1_CALM_RandomForest_N10_t3", "algo": "RandomForest", "regime": "CALM", "horizon": 1, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "3M_ret_5d", "IYM_BasicMaterials_ret_20d", "XLV_Health_zscore_60d", "EWG_Germany_ret_20d", "DE_Deere_ret_5d", "JNJ_ret_1d", "EQIX_Equinix_ret_5d", "ITT_ITTInc_ret_5d"], "is_new": true}, {"model_id": "new_h1_CALM_RandomForest_N10_t4", "algo": "RandomForest", "regime": "CALM", "horizon": 1, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "CTAS_Cintas_vol_20d", "TED_Spread_vol_20d", "LUV_SouthwestAir_ret_5d", "Brent_Oil_FRED_ret_20d", "MRK_Merck_zscore_60d", "spx_vol_5d", "MS_MorganStanley_zscore_60d", "VOD_Vodafone_zscore_60d"], "is_new": true}, {"model_id": "new_h1_CALM_RandomForest_N10_t5", "algo": "RandomForest", "regime": "CALM", "horizon": 1, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "DE_Deere_vol_20d", "EFFR_vol_20d", "ORCL_vol_20d", "heston_var_ev_h3", "SBUX_vol_20d", "hmm_p_stress", "INTC_ret_5d", "EOG_EOGResources_vol_20d"], "is_new": true}, {"model_id": "new_h1_CALM_RandomForest_N10_t6", "algo": "RandomForest", "regime": "CALM", "horizon": 1, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "MSTR_Bitcoin3_ret_1d", "HangSeng_HK_vol_20d", "HUM_Humana_ret_5d", "DAX_Germany_zscore_60d", "EQIX_Equinix_ret_5d", "3M_ret_5d", "US3Y_Rate_ret_5d", "MS_MorganStanley_ret_5d"], "is_new": true}, {"model_id": "new_h1_CALM_RandomForest_N10_t7", "algo": "RandomForest", "regime": "CALM", "horizon": 1, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "XLK_Tech_zscore_60d", "EWL_Switzerland_zscore_60d", "heston_var_ev_h3", "US3M_Rate_vol_20d", "CPB_CampbellSoup_vol_20d", "SJM_JM_Smucker_ret_1d", "DHR_vol_20d", "IYM_BasicMaterials_ret_20d"], "is_new": true}, {"model_id": "new_h1_CALM_RandomForest_N12_t0", "algo": "RandomForest", "regime": "CALM", "horizon": 1, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "INTC_ret_5d", "ENB_EnbridgeInc_ret_1d", "heston_ev_h3", "CLX_Clorox_vol_20d", "vix_mean_abs_ret_5d", "heston_var_ev_h3", "IYR_US_REIT2_zscore_60d", "PAYX_Paychex_vol_20d", "AMD_ret_1d", "FedFunds_zscore_60d"], "is_new": true}, {"model_id": "new_h1_CALM_RandomForest_N12_t1", "algo": "RandomForest", "regime": "CALM", "horizon": 1, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EXC_Exelon_ret_1d", "CPB_CampbellSoup_zscore_60d", "3M_ret_5d", "AXP_Amex_vol_20d", "MSTR_Bitcoin3_ret_5d", "ORCL_zscore_60d", "NVDA_vol_20d", "ENB_EnbridgeInc_ret_1d", "NFCI_ret_5d", "SCHW_Schwab_ret_5d"], "is_new": true}, {"model_id": "new_h1_CALM_RandomForest_N12_t2", "algo": "RandomForest", "regime": "CALM", "horizon": 1, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "Retail_Sales_zscore_60d", "US5Y_Rate_ret_5d", "hmm_p_stress", "EWM_Malaysia_ret_1d", "EMR_Emerson_ret_20d", "PAYX_Paychex_zscore_60d", "Brent_Oil_FRED_ret_20d", "ASX_Australia_ret_5d", "HUM_Humana_ret_5d", "AMT_AmericanTower_ret_1d"], "is_new": true}, {"model_id": "new_h1_CALM_RandomForest_N12_t3", "algo": "RandomForest", "regime": "CALM", "horizon": 1, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "JNJ_ret_1d", "EOG_EOGResources_ret_5d", "MSTR_Bitcoin3_ret_1d", "EMR_Emerson_ret_20d", "Brent_Oil_FRED_ret_5d", "T_ret_1d", "CMCSA_ret_1d", "Industrial_Production_zscore_60d", "VVIX_ret_20d", "XLV_Health_zscore_60d"], "is_new": true}, {"model_id": "new_h1_CALM_RandomForest_N12_t4", "algo": "RandomForest", "regime": "CALM", "horizon": 1, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "XLF_Fin_vol_20d", "ORCL_vol_20d", "EFFR_vol_20d", "BA_ret_1d", "CPB_CampbellSoup_ret_5d", "IYM_BasicMaterials_ret_20d", "EWG_Germany_vol_20d", "EWA_Australia_ret_1d", "NEE_NextEra_ret_20d", "US1Y_Rate_ret_20d"], "is_new": true}, {"model_id": "new_h1_CALM_RandomForest_N12_t5", "algo": "RandomForest", "regime": "CALM", "horizon": 1, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWQ_France_ret_20d", "CPB_CampbellSoup_ret_20d", "AXP_Amex_vol_20d", "MS_MorganStanley_ret_5d", "US3Y_Rate_ret_5d", "T_ret_1d", "SBUX_zscore_60d", "EFFR_ret_1d", "BA_ret_1d", "LOW_Lowes_ret_5d"], "is_new": true}, {"model_id": "new_h1_CALM_RandomForest_N12_t6", "algo": "RandomForest", "regime": "CALM", "horizon": 1, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "spx_vol_5d", "XOM_ret_20d", "WTI_Oil_FRED_zscore_60d", "Retail_Sales_zscore_60d", "EWG_Germany_ret_20d", "CPB_CampbellSoup_ret_20d", "AMZN_ret_5d", "EWG_Germany_vol_20d", "EOG_EOGResources_ret_5d", "heston_var_ev_h3"], "is_new": true}, {"model_id": "new_h1_CALM_RandomForest_N12_t7", "algo": "RandomForest", "regime": "CALM", "horizon": 1, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "HangSeng_HK_vol_20d", "MS_MorganStanley_zscore_60d", "PPL_PPL_ret_1d", "SBUX_zscore_60d", "spx_vol_5d", "FedFunds_zscore_60d", "CMCSA_ret_1d", "heston_var_ev_h5", "TXN_vol_20d", "MS_MorganStanley_ret_1d"], "is_new": true}, {"model_id": "new_h1_CALM_RandomForest_N15_t0", "algo": "RandomForest", "regime": "CALM", "horizon": 1, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "ES_Evergy_ret_1d", "PAYX_Paychex_vol_20d", "AXP_Amex_ret_20d", "FedFunds_zscore_60d", "ORCL_zscore_60d", "PLD_Prologis_ret_5d", "EWM_Malaysia_ret_1d", "GD_GeneralDynamics_zscore_60d", "US3M_Rate_vol_20d", "BLK_BlackRock_zscore_60d", "DOW_Price_zscore_60d", "vix_acceleration_1d", "CPB_CampbellSoup_ret_20d"], "is_new": true}, {"model_id": "new_h1_CALM_RandomForest_N15_t1", "algo": "RandomForest", "regime": "CALM", "horizon": 1, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "US1Y_Rate_ret_20d", "BLK_BlackRock_zscore_60d", "MS_MorganStanley_ret_1d", "T10Y2Y_Spread_ret_5d", "NVDA_vol_20d", "TM_Telephone_ret_1d", "QQQ_vol_20d", "US3Y_Rate_ret_5d", "EWQ_France_ret_20d", "HangSeng_HK_ret_1d", "CCI_CrownCastle_vol_20d", "MS_MorganStanley_zscore_60d", "ASX_Australia_ret_5d"], "is_new": true}, {"model_id": "new_h1_CALM_RandomForest_N15_t2", "algo": "RandomForest", "regime": "CALM", "horizon": 1, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EOG_EOGResources_vol_20d", "ENB_EnbridgeInc_ret_1d", "EWJ_Japan_vol_20d", "EWA_Australia_zscore_60d", "DAX_Germany_zscore_60d", "AVB_AvalonBay_zscore_60d", "NWL_Newell_ret_20d", "CCI_CrownCastle_vol_20d", "ORCL_vol_20d", "EWG_Germany_ret_20d", "XOM_ret_20d", "DIS_vol_20d", "VVIX_ret_20d"], "is_new": true}, {"model_id": "new_h1_CALM_RandomForest_N15_t3", "algo": "RandomForest", "regime": "CALM", "horizon": 1, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "MS_MorganStanley_ret_1d", "US1Y_Rate_ret_5d", "MS_MorganStanley_zscore_60d", "US1Y_Rate_ret_20d", "IWM_SmallCap_vol_20d", "GILD_Gilead_ret_20d", "T10Y2Y_Spread_ret_5d", "PFE_ret_1d", "AMD_ret_5d", "XOM_ret_1d", "VOD_Vodafone_zscore_60d", "NVDA_vol_20d", "CTAS_Cintas_vol_20d"], "is_new": true}, {"model_id": "new_h1_CALM_RandomForest_N15_t4", "algo": "RandomForest", "regime": "CALM", "horizon": 1, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "CPB_CampbellSoup_zscore_60d", "US3M_Rate_vol_20d", "CPB_CampbellSoup_ret_5d", "HD_ret_5d", "TM_Telephone_ret_1d", "LLY_zscore_60d", "NFCI_ret_5d", "FedFunds_zscore_60d", "MSTR_Bitcoin3_ret_5d", "spx_abs_ret_max_5d", "SBUX_vol_20d", "US3Y_Rate_ret_5d", "XLF_Fin_vol_20d"], "is_new": true}, {"model_id": "new_h1_CALM_RandomForest_N15_t5", "algo": "RandomForest", "regime": "CALM", "horizon": 1, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "Michigan_Sentiment_ret_20d", "DHR_vol_20d", "XLY_Disc_vol_20d", "XLF_Fin_vol_20d", "M_Macys_vol_20d", "ASX_Australia_vol_20d", "MS_MorganStanley_zscore_60d", "IYM_BasicMaterials_ret_20d", "EWQ_France_ret_20d", "AXP_Amex_vol_20d", "QQQ_vol_20d", "GD_GeneralDynamics_zscore_60d", "MRK_Merck_zscore_60d"], "is_new": true}, {"model_id": "new_h1_CALM_RandomForest_N15_t6", "algo": "RandomForest", "regime": "CALM", "horizon": 1, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "HD_ret_1d", "SBUX_zscore_60d", "CPB_CampbellSoup_ret_5d", "EWM_Malaysia_vol_20d", "DE_Deere_vol_20d", "HangSeng_HK_ret_1d", "PAYX_Paychex_ret_20d", "ITT_ITTInc_ret_5d", "GD_GeneralDynamics_zscore_60d", "NOC_Northrop_ret_20d", "XLB_Materials_zscore_60d", "heston_var_ev_h7", "CLX_Clorox_vol_20d"], "is_new": true}, {"model_id": "new_h1_CALM_RandomForest_N15_t7", "algo": "RandomForest", "regime": "CALM", "horizon": 1, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "SBUX_vol_20d", "SPY_zscore_60d", "MRK_Merck_zscore_60d", "BDX_Becton_Dickinson_ret_20d", "SO_SouthernCo_ret_5d", "EFFR_vol_20d", "BTI_BritishAmerican_ret_20d", "IBEX_Spain_ret_20d", "SLB_Schlumberger_ret_1d", "Michigan_Sentiment_ret_20d", "DE_Deere_vol_20d", "TED_Spread_vol_20d", "US3M_Rate_zscore_60d"], "is_new": true}, {"model_id": "new_h1_CALM_RandomForest_N20_t0", "algo": "RandomForest", "regime": "CALM", "horizon": 1, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "BDX_Becton_Dickinson_ret_20d", "EWY_Korea_ret_20d", "ASX_Australia_ret_5d", "PCAR_PaccarInc_ret_5d", "AXP_Amex_vol_20d", "LOW_Lowes_ret_5d", "EWC_Canada_zscore_60d", "GILD_Gilead_ret_20d", "MO_AltriaMG_ret_1d", "PG_ret_20d", "SLB_Schlumberger_ret_1d", "MS_MorganStanley_zscore_60d", "heston_ev_h3", "DAX_Germany_vol_20d", "AORD_AUS_zscore_60d", "EWQ_France_zscore_60d", "MRK_Merck_zscore_60d", "SBUX_zscore_60d"], "is_new": true}, {"model_id": "new_h1_CALM_RandomForest_N20_t1", "algo": "RandomForest", "regime": "CALM", "horizon": 1, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "VOD_Vodafone_zscore_60d", "US1Y_Rate_ret_20d", "LMT_LockheedMartin_ret_1d", "LLY_zscore_60d", "EMR_Emerson_ret_20d", "EXC_Exelon_zscore_60d", "CMCSA_ret_1d", "HD_ret_20d", "AORD_AUS_zscore_60d", "MSTR_Bitcoin3_ret_5d", "AXP_Amex_vol_20d", "US3Y_Rate_ret_5d", "EWJ_Japan_vol_20d", "EWL_Switzerland_zscore_60d", "heston_var_ev_h7", "EWM_Malaysia_ret_1d", "EWG_Germany_vol_20d", "GD_GeneralDynamics_zscore_60d"], "is_new": true}, {"model_id": "new_h1_CALM_RandomForest_N20_t2", "algo": "RandomForest", "regime": "CALM", "horizon": 1, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "QQQ_vol_20d", "DE_Deere_ret_5d", "heston_var_ev_h7", "EOG_EOGResources_vol_20d", "WTI_Oil_FRED_zscore_60d", "AMD_ret_1d", "EWM_Malaysia_ret_1d", "EWY_Korea_zscore_60d", "XLF_Fin_vol_20d", "HangSeng_HK_vol_20d", "heston_var_ev_h5", "GILD_Gilead_ret_20d", "NWL_Newell_ret_20d", "MSTR_Bitcoin3_ret_1d", "LMT_LockheedMartin_ret_1d", "PFE_ret_1d", "EWL_Switzerland_vol_20d", "PLD_Prologis_ret_5d"], "is_new": true}, {"model_id": "new_h1_CALM_RandomForest_N20_t3", "algo": "RandomForest", "regime": "CALM", "horizon": 1, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "SLB_Schlumberger_ret_5d", "spx_vol_5d", "MRK_Merck_zscore_60d", "US30Y_Rate_ret_20d", "heston_var_ev_h7", "VVIX_ret_20d", "XLF_Fin_vol_20d", "HD_ret_20d", "HangSeng_HK_ret_1d", "CPB_CampbellSoup_zscore_60d", "SBUX_ret_5d", "BTI_BritishAmerican_ret_20d", "Industrial_Production_zscore_60d", "GILD_Gilead_ret_20d", "heston_var_ev_h3", "IWM_SmallCap_vol_20d", "EXC_Exelon_zscore_60d", "spx_abs_ret_max_5d"], "is_new": true}, {"model_id": "new_h1_CALM_RandomForest_N20_t4", "algo": "RandomForest", "regime": "CALM", "horizon": 1, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "SPY_zscore_60d", "EWQ_France_zscore_60d", "BDX_Becton_Dickinson_ret_20d", "WTI_Oil_FRED_zscore_60d", "Michigan_Sentiment_ret_20d", "spx_vol_5d", "INTC_ret_5d", "DHR_ret_1d", "AMZN_ret_5d", "spx_abs_ret_max_5d", "SJM_JM_Smucker_ret_1d", "HUM_Humana_ret_5d", "EXC_Exelon_zscore_60d", "NVDA_vol_20d", "MS_MorganStanley_ret_1d", "ASX_Australia_vol_20d", "US7Y_Rate_ret_20d", "US3Y_Rate_ret_5d"], "is_new": true}, {"model_id": "new_h1_CALM_RandomForest_N20_t5", "algo": "RandomForest", "regime": "CALM", "horizon": 1, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "vix_mean_abs_ret_5d", "LOW_Lowes_ret_20d", "SLB_Schlumberger_ret_1d", "HangSeng_HK_ret_1d", "EOG_EOGResources_ret_5d", "spx_momentum_3d", "US1Y_Rate_ret_5d", "EQR_Equity_ret_1d", "HUM_Humana_ret_5d", "CPB_CampbellSoup_ret_20d", "PPL_PPL_ret_1d", "DAX_Germany_zscore_60d", "EXC_Exelon_zscore_60d", "US30Y_Rate_ret_20d", "IYR_US_REIT2_zscore_60d", "spx_abs_ret_max_5d", "CLX_Clorox_vol_20d", "EWL_Switzerland_zscore_60d"], "is_new": true}, {"model_id": "new_h1_CALM_RandomForest_N20_t6", "algo": "RandomForest", "regime": "CALM", "horizon": 1, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "IYM_BasicMaterials_ret_20d", "MRK_Merck_zscore_60d", "EXC_Exelon_zscore_60d", "US7Y_Rate_ret_20d", "HUM_Humana_ret_5d", "PG_ret_20d", "Core_PCE_zscore_60d", "PPL_PPL_ret_1d", "BA_ret_1d", "heston_ev_h3", "US6M_Rate_ret_20d", "XLY_Disc_vol_20d", "Brent_Oil_FRED_ret_20d", "EWA_Australia_zscore_60d", "SJM_JM_Smucker_ret_5d", "MSTR_Bitcoin3_ret_20d", "TED_Spread_zscore_60d", "HangSeng_HK_ret_1d"], "is_new": true}, {"model_id": "new_h1_CALM_RandomForest_N20_t7", "algo": "RandomForest", "regime": "CALM", "horizon": 1, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWG_Germany_ret_20d", "JNJ_ret_1d", "NEE_NextEra_ret_20d", "US3M_Rate_zscore_60d", "PFE_ret_1d", "DHR_vol_20d", "INTC_ret_1d", "EQR_Equity_ret_1d", "CCI_CrownCastle_vol_20d", "SBUX_vol_20d", "IYR_US_REIT2_zscore_60d", "gjr_condvar_h1", "EQIX_Equinix_ret_5d", "heston_ev_h3", "BDX_Becton_Dickinson_ret_20d", "3M_vol_20d", "PAYX_Paychex_zscore_60d", "GILD_Gilead_ret_20d"], "is_new": true}, {"model_id": "new_h1_CALM_RandomForest_N25_t0", "algo": "RandomForest", "regime": "CALM", "horizon": 1, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "CLX_Clorox_vol_20d", "DE_Deere_ret_5d", "XLV_Health_zscore_60d", "ORCL_vol_20d", "heston_ev_h3", "Nikkei_Japan_vol_20d", "Michigan_Sentiment_ret_20d", "hmm_p_stress", "Retail_Sales_zscore_60d", "MSTR_Bitcoin3_ret_5d", "3M_vol_20d", "ASX_Australia_vol_20d", "TGT_Target_zscore_60d", "EWG_Germany_ret_20d", "MRK_Merck_zscore_60d", "AMD_ret_5d", "ITT_ITTInc_ret_5d", "TED_Spread_zscore_60d", "EWA_Australia_ret_1d", "AMGN_Amgen_ret_1d", "gjr_condvar_h1", "EWY_Korea_ret_20d", "EWM_Malaysia_ret_1d"], "is_new": true}, {"model_id": "new_h1_CALM_RandomForest_N25_t1", "algo": "RandomForest", "regime": "CALM", "horizon": 1, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "CPB_CampbellSoup_vol_20d", "MSTR_Bitcoin3_ret_5d", "SJM_JM_Smucker_ret_5d", "VOD_Vodafone_zscore_60d", "HangSeng_HK_vol_20d", "ENB_EnbridgeInc_ret_1d", "BTI_BritishAmerican_ret_20d", "AXP_Amex_ret_20d", "HangSeng_HK_ret_1d", "HD_zscore_60d", "PCAR_PaccarInc_ret_5d", "T10Y2Y_Spread_ret_5d", "VVIX_ret_20d", "TED_Spread_vol_20d", "Industrial_Production_zscore_60d", "ORCL_vol_20d", "DE_Deere_vol_20d", "JNJ_ret_1d", "SBUX_zscore_60d", "LOW_Lowes_ret_20d", "EWA_Australia_ret_1d", "hmm_p_stress", "Michigan_Sentiment_ret_20d"], "is_new": true}, {"model_id": "new_h1_CALM_RandomForest_N25_t2", "algo": "RandomForest", "regime": "CALM", "horizon": 1, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWJ_Japan_vol_20d", "AMT_AmericanTower_ret_1d", "VOD_Vodafone_zscore_60d", "SLB_Schlumberger_ret_1d", "EOG_EOGResources_ret_5d", "DAX_Germany_vol_20d", "LUV_SouthwestAir_ret_5d", "ASX_Australia_vol_20d", "MSTR_Bitcoin3_ret_1d", "EXC_Exelon_ret_1d", "TGT_Target_zscore_60d", "XOM_ret_1d", "AXP_Amex_vol_20d", "VVIX_ret_20d", "XOM_ret_20d", "CPB_CampbellSoup_zscore_60d", "vix_acceleration_1d", "XLV_Health_zscore_60d", "Core_CPI_zscore_60d", "NOC_Northrop_ret_20d", "PAYX_Paychex_ret_20d", "MSTR_Bitcoin3_ret_20d", "hmm_p_stress"], "is_new": true}, {"model_id": "new_h1_CALM_RandomForest_N25_t3", "algo": "RandomForest", "regime": "CALM", "horizon": 1, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWG_Germany_vol_20d", "gjr_condvar_h1", "AMD_ret_1d", "HangSeng_HK_ret_5d", "NVDA_vol_20d", "CPB_CampbellSoup_ret_5d", "T10Y2Y_Spread_ret_5d", "GE_ret_1d", "XOM_ret_1d", "XLF_Fin_vol_20d", "DHR_ret_1d", "spx_abs_ret_max_5d", "CPB_CampbellSoup_vol_20d", "SLB_Schlumberger_ret_5d", "Core_PCE_zscore_60d", "HD_ret_20d", "Core_CPI_zscore_60d", "CPB_CampbellSoup_zscore_60d", "SJM_JM_Smucker_ret_1d", "EWC_Canada_zscore_60d", "EWG_Germany_ret_20d", "FedFunds_zscore_60d", "heston_var_ev_h7"], "is_new": true}, {"model_id": "new_h1_CALM_RandomForest_N25_t4", "algo": "RandomForest", "regime": "CALM", "horizon": 1, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EMR_Emerson_ret_20d", "AVB_AvalonBay_zscore_60d", "T_ret_1d", "SJM_JM_Smucker_ret_1d", "EWH_HongKong_ret_5d", "SBUX_ret_5d", "IYM_BasicMaterials_ret_20d", "EWQ_France_ret_20d", "3M_ret_5d", "AMD_ret_5d", "CLX_Clorox_vol_20d", "CTAS_Cintas_vol_20d", "US3Y_Rate_ret_5d", "SCHW_Schwab_ret_5d", "DOW_Price_zscore_60d", "CMCSA_ret_1d", "SLB_Schlumberger_ret_5d", "Brent_Oil_FRED_ret_5d", "CPB_CampbellSoup_ret_5d", "EXC_Exelon_ret_1d", "EWM_Malaysia_zscore_60d", "PCAR_PaccarInc_ret_5d", "HUM_Humana_ret_5d"], "is_new": true}, {"model_id": "new_h1_CALM_RandomForest_N25_t5", "algo": "RandomForest", "regime": "CALM", "horizon": 1, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EOG_EOGResources_vol_20d", "AVB_AvalonBay_zscore_60d", "HD_zscore_60d", "LOW_Lowes_ret_5d", "INTC_ret_1d", "Nikkei_Japan_vol_20d", "Core_CPI_zscore_60d", "CPB_CampbellSoup_ret_20d", "AXP_Amex_ret_20d", "AMD_ret_1d", "DHR_ret_1d", "EOG_EOGResources_ret_5d", "IYM_BasicMaterials_ret_20d", "LOW_Lowes_ret_20d", "ORCL_zscore_60d", "GD_GeneralDynamics_zscore_60d", "CMCSA_ret_1d", "MRK_Merck_zscore_60d", "JNJ_ret_1d", "EQR_Equity_ret_1d", "heston_var_ev_h7", "M_Macys_vol_20d", "VOD_Vodafone_zscore_60d"], "is_new": true}, {"model_id": "new_h1_CALM_RandomForest_N25_t6", "algo": "RandomForest", "regime": "CALM", "horizon": 1, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "NVDA_vol_20d", "hmm_p_stress", "ENB_EnbridgeInc_ret_1d", "CTAS_Cintas_vol_20d", "DE_Deere_ret_5d", "BLK_BlackRock_zscore_60d", "EFFR_vol_20d", "CPB_CampbellSoup_vol_20d", "NOC_Northrop_ret_20d", "VRP_ma5", "SBUX_ret_5d", "US3M_Rate_zscore_60d", "ORCL_zscore_60d", "EQR_Equity_ret_1d", "XLK_Tech_zscore_60d", "MSTR_Bitcoin3_ret_20d", "LOW_Lowes_ret_20d", "AORD_AUS_zscore_60d", "spx_abs_ret_max_5d", "gjr_condvar_h1", "SBUX_vol_20d", "spx_momentum_3d", "NFCI_ret_5d"], "is_new": true}, {"model_id": "new_h1_CALM_RandomForest_N25_t7", "algo": "RandomForest", "regime": "CALM", "horizon": 1, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "spx_abs_ret_max_5d", "INTC_ret_1d", "AMD_ret_5d", "TED_Spread_vol_20d", "US5Y_Rate_ret_5d", "US3M_Rate_vol_20d", "EWM_Malaysia_ret_1d", "Industrial_Production_zscore_60d", "SPY_zscore_60d", "T10Y2Y_Spread_ret_5d", "XLY_Disc_vol_20d", "CPB_CampbellSoup_ret_20d", "Nikkei_Japan_zscore_60d", "EWY_Korea_ret_20d", "SBUX_ret_5d", "LLY_zscore_60d", "MSTR_Bitcoin3_ret_5d", "DOW_Price_zscore_60d", "US1Y_Rate_ret_5d", "BA_ret_1d", "AMD_ret_1d", "EFFR_vol_20d", "SLB_Schlumberger_ret_1d"], "is_new": true}, {"model_id": "new_h1_CALM_RandomForest_N30_t0", "algo": "RandomForest", "regime": "CALM", "horizon": 1, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "SBUX_zscore_60d", "ITT_ITTInc_ret_5d", "EWQ_France_zscore_60d", "IYM_BasicMaterials_ret_20d", "HD_ret_1d", "AXP_Amex_vol_20d", "SLB_Schlumberger_ret_1d", "EWM_Malaysia_ret_1d", "T_ret_1d", "MSTR_Bitcoin3_ret_1d", "Michigan_Sentiment_ret_20d", "EWQ_France_ret_20d", "CCI_CrownCastle_vol_20d", "PFE_ret_1d", "PAYX_Paychex_ret_20d", "US3Y_Rate_ret_5d", "PAYX_Paychex_vol_20d", "CLX_Clorox_vol_20d", "SJM_JM_Smucker_ret_1d", "XLF_Fin_vol_20d", "INTC_ret_1d", "Core_PCE_zscore_60d", "WTI_Oil_FRED_zscore_60d", "AMGN_Amgen_ret_1d", "CPB_CampbellSoup_vol_20d", "gjr_condvar_h1", "NFCI_ret_5d", "ASX_Australia_vol_20d"], "is_new": true}, {"model_id": "new_h1_CALM_RandomForest_N30_t1", "algo": "RandomForest", "regime": "CALM", "horizon": 1, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "SLB_Schlumberger_ret_5d", "US30Y_Rate_ret_20d", "EWG_Germany_vol_20d", "spx_abs_ret_max_5d", "PCAR_PaccarInc_ret_5d", "JNJ_ret_1d", "LUV_SouthwestAir_ret_5d", "BA_ret_1d", "EWM_Malaysia_ret_1d", "spx_momentum_3d", "IYM_BasicMaterials_ret_20d", "vix_acceleration_1d", "XLY_Disc_vol_20d", "WTI_Oil_FRED_zscore_60d", "QQQ_vol_20d", "heston_var_ev_h7", "PAYX_Paychex_zscore_60d", "Core_CPI_zscore_60d", "CPB_CampbellSoup_ret_20d", "EWL_Switzerland_zscore_60d", "3M_ret_5d", "hmm_p_stress", "LMT_LockheedMartin_vol_20d", "HD_zscore_60d", "T_ret_1d", "T10Y2Y_Spread_ret_5d", "MRK_Merck_zscore_60d", "CPB_CampbellSoup_zscore_60d"], "is_new": true}, {"model_id": "new_h1_CALM_RandomForest_N30_t2", "algo": "RandomForest", "regime": "CALM", "horizon": 1, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "NEE_NextEra_ret_20d", "CPB_CampbellSoup_zscore_60d", "EXC_Exelon_ret_1d", "XLK_Tech_zscore_60d", "EWC_Canada_zscore_60d", "EWA_Australia_ret_1d", "Core_PCE_zscore_60d", "MS_MorganStanley_ret_5d", "EWL_Switzerland_vol_20d", "PFE_ret_1d", "hmm_p_stress", "Industrial_Production_zscore_60d", "vix_acceleration_1d", "NOC_Northrop_ret_20d", "AXP_Amex_ret_20d", "VOD_Vodafone_zscore_60d", "DAX_Germany_vol_20d", "PAYX_Paychex_zscore_60d", "heston_ev_h3", "spx_momentum_3d", "SLB_Schlumberger_ret_1d", "DE_Deere_vol_20d", "SJM_JM_Smucker_ret_1d", "3M_vol_20d", "TED_Spread_zscore_60d", "AXP_Amex_vol_20d", "US1Y_Rate_ret_20d", "Nikkei_Japan_zscore_60d"], "is_new": true}, {"model_id": "new_h1_CALM_RandomForest_N30_t3", "algo": "RandomForest", "regime": "CALM", "horizon": 1, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "DAX_Germany_zscore_60d", "CPB_CampbellSoup_vol_20d", "MS_MorganStanley_ret_1d", "spx_momentum_3d", "WTI_Oil_FRED_zscore_60d", "XLF_Fin_vol_20d", "ITT_ITTInc_ret_5d", "SLB_Schlumberger_ret_1d", "MSTR_Bitcoin3_ret_5d", "NVDA_vol_20d", "hmm_p_stress", "EWY_Korea_ret_20d", "GILD_Gilead_ret_20d", "MS_MorganStanley_zscore_60d", "JNJ_ret_1d", "HUM_Humana_ret_5d", "PAYX_Paychex_zscore_60d", "DE_Deere_vol_20d", "DHR_ret_1d", "NFCI_ret_5d", "SO_SouthernCo_ret_5d", "HangSeng_HK_ret_1d", "T_ret_1d", "INTC_ret_1d", "US1Y_Rate_ret_20d", "EWL_Switzerland_vol_20d", "EWG_Germany_ret_20d", "EWH_HongKong_ret_5d"], "is_new": true}, {"model_id": "new_h1_CALM_RandomForest_N30_t4", "algo": "RandomForest", "regime": "CALM", "horizon": 1, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "hmm_p_stress", "EQIX_Equinix_ret_5d", "SJM_JM_Smucker_ret_5d", "TM_Telephone_vol_20d", "heston_var_ev_h3", "ITT_ITTInc_ret_5d", "Core_CPI_zscore_60d", "LMT_LockheedMartin_vol_20d", "US30Y_Rate_ret_20d", "Retail_Sales_zscore_60d", "US7Y_Rate_ret_20d", "US3Y_Rate_ret_5d", "NOC_Northrop_ret_20d", "TGT_Target_zscore_60d", "HD_ret_5d", "NWL_Newell_ret_20d", "AMD_ret_5d", "T10Y2Y_Spread_ret_5d", "CPB_CampbellSoup_zscore_60d", "AMZN_ret_5d", "MSTR_Bitcoin3_ret_20d", "DIS_vol_20d", "BA_ret_1d", "BDX_Becton_Dickinson_ret_20d", "EWG_Germany_ret_20d", "GD_GeneralDynamics_zscore_60d", "EMR_Emerson_ret_20d", "XLV_Health_zscore_60d"], "is_new": true}, {"model_id": "new_h1_CALM_RandomForest_N30_t5", "algo": "RandomForest", "regime": "CALM", "horizon": 1, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "HUM_Humana_ret_5d", "EWG_Germany_vol_20d", "TM_Telephone_vol_20d", "ASX_Australia_ret_5d", "LOW_Lowes_ret_20d", "PAYX_Paychex_zscore_60d", "CPB_CampbellSoup_vol_20d", "XLV_Health_zscore_60d", "US30Y_Rate_ret_20d", "EWM_Malaysia_ret_1d", "heston_var_ev_h3", "MS_MorganStanley_ret_1d", "EWQ_France_ret_20d", "SLB_Schlumberger_ret_5d", "HD_ret_5d", "AXP_Amex_vol_20d", "MSTR_Bitcoin3_ret_20d", "XOM_ret_20d", "NOC_Northrop_ret_20d", "INTC_ret_1d", "US1Y_Rate_ret_20d", "BA_ret_1d", "AMD_ret_5d", "EWY_Korea_zscore_60d", "EWG_Germany_ret_20d", "PAYX_Paychex_ret_20d", "heston_ev_h3", "MO_AltriaMG_ret_1d"], "is_new": true}, {"model_id": "new_h1_CALM_RandomForest_N30_t6", "algo": "RandomForest", "regime": "CALM", "horizon": 1, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "PAYX_Paychex_vol_20d", "EQIX_Equinix_ret_5d", "Brent_Oil_FRED_ret_5d", "ES_Evergy_ret_1d", "EOG_EOGResources_vol_20d", "SO_SouthernCo_ret_5d", "Core_PCE_zscore_60d", "MSTR_Bitcoin3_ret_1d", "AVB_AvalonBay_zscore_60d", "Nikkei_Japan_vol_20d", "HD_ret_1d", "US1Y_Rate_ret_5d", "FedFunds_zscore_60d", "DE_Deere_ret_5d", "EMR_Emerson_ret_20d", "US7Y_Rate_ret_20d", "EXC_Exelon_ret_1d", "3M_vol_20d", "IYM_BasicMaterials_ret_20d", "PLD_Prologis_ret_5d", "EFFR_ret_1d", "heston_var_ev_h3", "VRP_ma5", "ITT_ITTInc_ret_5d", "BA_ret_1d", "HangSeng_HK_ret_5d", "SBUX_zscore_60d", "T_ret_1d"], "is_new": true}, {"model_id": "new_h1_CALM_RandomForest_N30_t7", "algo": "RandomForest", "regime": "CALM", "horizon": 1, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "ES_Evergy_ret_1d", "T10Y2Y_Spread_ret_5d", "BTI_BritishAmerican_ret_5d", "SLB_Schlumberger_ret_1d", "EFFR_vol_20d", "NOC_Northrop_ret_20d", "TXN_vol_20d", "TM_Telephone_ret_1d", "Michigan_Sentiment_ret_20d", "CMCSA_ret_1d", "LMT_LockheedMartin_ret_1d", "EWL_Switzerland_zscore_60d", "EWJ_Japan_vol_20d", "IYM_BasicMaterials_ret_20d", "XLY_Disc_vol_20d", "LOW_Lowes_ret_5d", "HUM_Humana_ret_5d", "EWL_Switzerland_vol_20d", "TED_Spread_zscore_60d", "QQQ_vol_20d", "US30Y_Rate_ret_20d", "AMD_ret_1d", "Core_CPI_zscore_60d", "LLY_zscore_60d", "PAYX_Paychex_vol_20d", "EWQ_France_zscore_60d", "EWS_Singapore_ret_5d", "AVB_AvalonBay_zscore_60d"], "is_new": true}, {"model_id": "new_h1_CALM_LogisticRegression_N5_t0", "algo": "LogisticRegression", "regime": "CALM", "horizon": 1, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "MSTR_Bitcoin3_ret_5d", "EXC_Exelon_ret_1d", "TM_Telephone_vol_20d"], "is_new": true}, {"model_id": "new_h1_CALM_LogisticRegression_N5_t1", "algo": "LogisticRegression", "regime": "CALM", "horizon": 1, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "SO_SouthernCo_ret_5d", "SLB_Schlumberger_ret_5d", "PAYX_Paychex_vol_20d"], "is_new": true}, {"model_id": "new_h1_CALM_LogisticRegression_N5_t2", "algo": "LogisticRegression", "regime": "CALM", "horizon": 1, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "ITT_ITTInc_ret_5d", "SO_SouthernCo_ret_5d", "CI_Cigna_vol_20d"], "is_new": true}, {"model_id": "new_h1_CALM_LogisticRegression_N5_t3", "algo": "LogisticRegression", "regime": "CALM", "horizon": 1, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWH_HongKong_ret_5d", "vix_mean_abs_ret_5d", "gjr_condvar_h1"], "is_new": true}, {"model_id": "new_h1_CALM_LogisticRegression_N5_t4", "algo": "LogisticRegression", "regime": "CALM", "horizon": 1, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "TM_Telephone_ret_1d", "ORCL_vol_20d", "heston_var_ev_h7"], "is_new": true}, {"model_id": "new_h1_CALM_LogisticRegression_N5_t5", "algo": "LogisticRegression", "regime": "CALM", "horizon": 1, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "IYM_BasicMaterials_ret_20d", "ASX_Australia_vol_20d", "CLX_Clorox_vol_20d"], "is_new": true}, {"model_id": "new_h1_CALM_LogisticRegression_N5_t6", "algo": "LogisticRegression", "regime": "CALM", "horizon": 1, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "US1Y_Rate_ret_5d", "Brent_Oil_FRED_ret_20d", "US5Y_Rate_ret_5d"], "is_new": true}, {"model_id": "new_h1_CALM_LogisticRegression_N5_t7", "algo": "LogisticRegression", "regime": "CALM", "horizon": 1, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "PAYX_Paychex_zscore_60d", "gjr_condvar_h1", "Michigan_Sentiment_ret_20d"], "is_new": true}, {"model_id": "new_h1_CALM_LogisticRegression_N8_t0", "algo": "LogisticRegression", "regime": "CALM", "horizon": 1, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "Core_PCE_zscore_60d", "TGT_Target_zscore_60d", "3M_vol_20d", "heston_var_ev_h5", "XLY_Disc_vol_20d", "BTI_BritishAmerican_ret_20d"], "is_new": true}, {"model_id": "new_h1_CALM_LogisticRegression_N8_t1", "algo": "LogisticRegression", "regime": "CALM", "horizon": 1, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "SJM_JM_Smucker_ret_5d", "CPB_CampbellSoup_zscore_60d", "HD_zscore_60d", "HangSeng_HK_ret_5d", "XLF_Fin_vol_20d", "CLX_Clorox_vol_20d"], "is_new": true}, {"model_id": "new_h1_CALM_LogisticRegression_N8_t2", "algo": "LogisticRegression", "regime": "CALM", "horizon": 1, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "spx_vol_5d", "AORD_AUS_zscore_60d", "QQQ_vol_20d", "EFFR_vol_20d", "TGT_Target_zscore_60d", "EWL_Switzerland_zscore_60d"], "is_new": true}, {"model_id": "new_h1_CALM_LogisticRegression_N8_t3", "algo": "LogisticRegression", "regime": "CALM", "horizon": 1, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "HangSeng_HK_ret_1d", "heston_var_ev_h7", "MRK_Merck_zscore_60d", "AMD_ret_1d", "US1Y_Rate_ret_20d", "US1Y_Rate_ret_5d"], "is_new": true}, {"model_id": "new_h1_CALM_LogisticRegression_N8_t4", "algo": "LogisticRegression", "regime": "CALM", "horizon": 1, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "US3Y_Rate_ret_5d", "VVIX_ret_20d", "MS_MorganStanley_ret_5d", "TM_Telephone_ret_1d", "T_ret_1d", "CI_Cigna_vol_20d"], "is_new": true}, {"model_id": "new_h1_CALM_LogisticRegression_N8_t5", "algo": "LogisticRegression", "regime": "CALM", "horizon": 1, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "NFCI_ret_5d", "heston_var_ev_h7", "SO_SouthernCo_ret_5d", "DHR_vol_20d", "EMR_Emerson_ret_20d", "ES_Evergy_ret_1d"], "is_new": true}, {"model_id": "new_h1_CALM_LogisticRegression_N8_t6", "algo": "LogisticRegression", "regime": "CALM", "horizon": 1, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EQR_Equity_ret_1d", "vix_mean_abs_ret_5d", "US3M_Rate_zscore_60d", "BA_ret_1d", "T_ret_1d", "PG_ret_20d"], "is_new": true}, {"model_id": "new_h1_CALM_LogisticRegression_N8_t7", "algo": "LogisticRegression", "regime": "CALM", "horizon": 1, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWY_Korea_zscore_60d", "CPB_CampbellSoup_ret_5d", "EWC_Canada_zscore_60d", "US6M_Rate_ret_20d", "SBUX_vol_20d", "IYM_BasicMaterials_ret_20d"], "is_new": true}, {"model_id": "new_h1_CALM_LogisticRegression_N10_t0", "algo": "LogisticRegression", "regime": "CALM", "horizon": 1, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "HD_ret_20d", "XLY_Disc_vol_20d", "BTI_BritishAmerican_ret_5d", "EWA_Australia_zscore_60d", "INTC_ret_5d", "SBUX_vol_20d", "PAYX_Paychex_zscore_60d", "IYM_BasicMaterials_ret_20d"], "is_new": true}, {"model_id": "new_h1_CALM_LogisticRegression_N10_t1", "algo": "LogisticRegression", "regime": "CALM", "horizon": 1, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "INTC_ret_1d", "Nikkei_Japan_vol_20d", "EOG_EOGResources_ret_5d", "3M_vol_20d", "EWY_Korea_ret_20d", "EWL_Switzerland_zscore_60d", "heston_var_ev_h5", "ORCL_vol_20d"], "is_new": true}, {"model_id": "new_h1_CALM_LogisticRegression_N10_t2", "algo": "LogisticRegression", "regime": "CALM", "horizon": 1, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "Retail_Sales_zscore_60d", "PG_ret_20d", "PAYX_Paychex_vol_20d", "LOW_Lowes_ret_20d", "HangSeng_HK_vol_20d", "SBUX_vol_20d", "US1Y_Rate_ret_20d", "CPB_CampbellSoup_zscore_60d"], "is_new": true}, {"model_id": "new_h1_CALM_LogisticRegression_N10_t3", "algo": "LogisticRegression", "regime": "CALM", "horizon": 1, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "TM_Telephone_ret_1d", "EWC_Canada_zscore_60d", "FedFunds_zscore_60d", "XOM_ret_20d", "Brent_Oil_FRED_ret_5d", "AMGN_Amgen_ret_1d", "SLB_Schlumberger_ret_1d", "gjr_condvar_h1"], "is_new": true}, {"model_id": "new_h1_CALM_LogisticRegression_N10_t4", "algo": "LogisticRegression", "regime": "CALM", "horizon": 1, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "US1Y_Rate_ret_20d", "EXC_Exelon_zscore_60d", "TXN_vol_20d", "TM_Telephone_vol_20d", "PPL_PPL_ret_1d", "spx_abs_ret_max_5d", "DOW_Price_zscore_60d", "LUV_SouthwestAir_ret_5d"], "is_new": true}, {"model_id": "new_h1_CALM_LogisticRegression_N10_t5", "algo": "LogisticRegression", "regime": "CALM", "horizon": 1, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "PG_ret_20d", "SJM_JM_Smucker_ret_5d", "SLB_Schlumberger_ret_5d", "CMCSA_ret_1d", "MS_MorganStanley_zscore_60d", "LUV_SouthwestAir_ret_5d", "AMD_ret_1d", "spx_vol_5d"], "is_new": true}, {"model_id": "new_h1_CALM_LogisticRegression_N10_t6", "algo": "LogisticRegression", "regime": "CALM", "horizon": 1, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EFFR_vol_20d", "EQIX_Equinix_ret_5d", "US3M_Rate_vol_20d", "spx_momentum_3d", "DIS_vol_20d", "SBUX_ret_5d", "LOW_Lowes_ret_20d", "AVB_AvalonBay_zscore_60d"], "is_new": true}, {"model_id": "new_h1_CALM_LogisticRegression_N10_t7", "algo": "LogisticRegression", "regime": "CALM", "horizon": 1, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "FedFunds_zscore_60d", "MO_AltriaMG_ret_1d", "EWM_Malaysia_zscore_60d", "AVB_AvalonBay_zscore_60d", "vix_mean_abs_ret_5d", "EWG_Germany_vol_20d", "SPY_zscore_60d", "LOW_Lowes_ret_5d"], "is_new": true}, {"model_id": "new_h1_CALM_LogisticRegression_N12_t0", "algo": "LogisticRegression", "regime": "CALM", "horizon": 1, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "AVB_AvalonBay_zscore_60d", "hmm_p_stress", "ITT_ITTInc_ret_5d", "CPB_CampbellSoup_vol_20d", "T10Y2Y_Spread_ret_5d", "heston_ev_h3", "GILD_Gilead_ret_20d", "FedFunds_zscore_60d", "US6M_Rate_ret_20d", "TXN_vol_20d"], "is_new": true}, {"model_id": "new_h1_CALM_LogisticRegression_N12_t1", "algo": "LogisticRegression", "regime": "CALM", "horizon": 1, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "LUV_SouthwestAir_ret_5d", "VOD_Vodafone_zscore_60d", "spx_abs_ret_max_5d", "Brent_Oil_FRED_ret_5d", "SPY_zscore_60d", "EWY_Korea_zscore_60d", "MSTR_Bitcoin3_ret_1d", "EWM_Malaysia_ret_1d", "Michigan_Sentiment_ret_20d", "LMT_LockheedMartin_vol_20d"], "is_new": true}, {"model_id": "new_h1_CALM_LogisticRegression_N12_t2", "algo": "LogisticRegression", "regime": "CALM", "horizon": 1, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "SJM_JM_Smucker_ret_5d", "DHR_ret_1d", "ORCL_zscore_60d", "ENB_EnbridgeInc_ret_1d", "EWM_Malaysia_vol_20d", "Brent_Oil_FRED_ret_5d", "SO_SouthernCo_ret_5d", "heston_var_ev_h5", "HD_ret_1d", "EWJ_Japan_vol_20d"], "is_new": true}, {"model_id": "new_h1_CALM_LogisticRegression_N12_t3", "algo": "LogisticRegression", "regime": "CALM", "horizon": 1, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "XLB_Materials_zscore_60d", "Core_PCE_zscore_60d", "spx_vol_5d", "EWM_Malaysia_vol_20d", "NOC_Northrop_ret_20d", "IYM_BasicMaterials_ret_20d", "Core_CPI_zscore_60d", "SBUX_ret_5d", "Retail_Sales_zscore_60d", "EWH_HongKong_ret_5d"], "is_new": true}, {"model_id": "new_h1_CALM_LogisticRegression_N12_t4", "algo": "LogisticRegression", "regime": "CALM", "horizon": 1, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWS_Singapore_ret_5d", "AXP_Amex_ret_20d", "DAX_Germany_zscore_60d", "US3Y_Rate_ret_5d", "VRP_ma5", "EMR_Emerson_ret_20d", "DE_Deere_vol_20d", "gjr_condvar_h1", "SO_SouthernCo_ret_5d", "AVB_AvalonBay_zscore_60d"], "is_new": true}, {"model_id": "new_h1_CALM_LogisticRegression_N12_t5", "algo": "LogisticRegression", "regime": "CALM", "horizon": 1, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EXC_Exelon_zscore_60d", "EQR_Equity_ret_1d", "SPY_zscore_60d", "Michigan_Sentiment_ret_20d", "WTI_Oil_FRED_zscore_60d", "US3M_Rate_vol_20d", "heston_var_ev_h3", "AMGN_Amgen_ret_1d", "spx_vol_5d", "HD_ret_20d"], "is_new": true}, {"model_id": "new_h1_CALM_LogisticRegression_N12_t6", "algo": "LogisticRegression", "regime": "CALM", "horizon": 1, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "SBUX_ret_5d", "EOG_EOGResources_ret_5d", "EWL_Switzerland_zscore_60d", "NFCI_ret_5d", "WTI_Oil_FRED_zscore_60d", "DHR_vol_20d", "US7Y_Rate_ret_20d", "EQR_Equity_ret_1d", "XOM_ret_1d", "spx_momentum_3d"], "is_new": true}, {"model_id": "new_h1_CALM_LogisticRegression_N12_t7", "algo": "LogisticRegression", "regime": "CALM", "horizon": 1, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "PCAR_PaccarInc_ret_5d", "VVIX_ret_20d", "VRP_ma5", "BTI_BritishAmerican_ret_5d", "EWM_Malaysia_vol_20d", "MSTR_Bitcoin3_ret_1d", "MSTR_Bitcoin3_ret_5d", "DE_Deere_ret_5d", "CPB_CampbellSoup_zscore_60d", "AMGN_Amgen_ret_1d"], "is_new": true}, {"model_id": "new_h1_CALM_LogisticRegression_N15_t0", "algo": "LogisticRegression", "regime": "CALM", "horizon": 1, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EQIX_Equinix_ret_5d", "QQQ_vol_20d", "XLV_Health_zscore_60d", "WTI_Oil_FRED_zscore_60d", "ASX_Australia_vol_20d", "TGT_Target_zscore_60d", "SJM_JM_Smucker_ret_5d", "Core_PCE_zscore_60d", "CMCSA_ret_1d", "SBUX_ret_5d", "spx_momentum_3d", "DE_Deere_ret_5d", "XLB_Materials_zscore_60d"], "is_new": true}, {"model_id": "new_h1_CALM_LogisticRegression_N15_t1", "algo": "LogisticRegression", "regime": "CALM", "horizon": 1, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "MO_AltriaMG_ret_1d", "AXP_Amex_ret_20d", "TED_Spread_zscore_60d", "Nikkei_Japan_vol_20d", "EWM_Malaysia_vol_20d", "SBUX_ret_5d", "MSTR_Bitcoin3_ret_20d", "LMT_LockheedMartin_ret_1d", "VOD_Vodafone_zscore_60d", "US5Y_Rate_ret_5d", "CPB_CampbellSoup_ret_5d", "NWL_Newell_ret_20d", "DE_Deere_vol_20d"], "is_new": true}, {"model_id": "new_h1_CALM_LogisticRegression_N15_t2", "algo": "LogisticRegression", "regime": "CALM", "horizon": 1, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "HD_ret_20d", "WTI_Oil_FRED_zscore_60d", "TXN_vol_20d", "NEE_NextEra_ret_20d", "EXC_Exelon_ret_1d", "EWA_Australia_ret_1d", "3M_vol_20d", "US5Y_Rate_ret_5d", "T10Y2Y_Spread_ret_5d", "Core_PCE_zscore_60d", "MSTR_Bitcoin3_ret_1d", "spx_abs_ret_max_5d", "EWG_Germany_vol_20d"], "is_new": true}, {"model_id": "new_h1_CALM_LogisticRegression_N15_t3", "algo": "LogisticRegression", "regime": "CALM", "horizon": 1, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "CPB_CampbellSoup_zscore_60d", "EFFR_ret_1d", "Nikkei_Japan_vol_20d", "hmm_p_stress", "US7Y_Rate_ret_20d", "EWY_Korea_zscore_60d", "EXC_Exelon_zscore_60d", "TM_Telephone_ret_1d", "NWL_Newell_ret_20d", "US3M_Rate_vol_20d", "LUV_SouthwestAir_ret_5d", "XLK_Tech_zscore_60d", "GILD_Gilead_ret_20d"], "is_new": true}, {"model_id": "new_h1_CALM_LogisticRegression_N15_t4", "algo": "LogisticRegression", "regime": "CALM", "horizon": 1, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "US1Y_Rate_ret_20d", "US1Y_Rate_ret_5d", "SBUX_ret_5d", "EQIX_Equinix_ret_5d", "MO_AltriaMG_ret_1d", "US30Y_Rate_ret_20d", "EWY_Korea_zscore_60d", "HangSeng_HK_vol_20d", "US5Y_Rate_ret_5d", "LLY_zscore_60d", "MS_MorganStanley_zscore_60d", "DHR_ret_1d", "spx_momentum_3d"], "is_new": true}, {"model_id": "new_h1_CALM_LogisticRegression_N15_t5", "algo": "LogisticRegression", "regime": "CALM", "horizon": 1, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "CPB_CampbellSoup_ret_5d", "EWJ_Japan_vol_20d", "SLB_Schlumberger_ret_5d", "M_Macys_vol_20d", "Industrial_Production_zscore_60d", "T_ret_1d", "XLV_Health_zscore_60d", "BA_ret_1d", "EMR_Emerson_ret_20d", "MS_MorganStanley_ret_1d", "heston_ev_h3", "3M_vol_20d", "CMCSA_ret_1d"], "is_new": true}, {"model_id": "new_h1_CALM_LogisticRegression_N15_t6", "algo": "LogisticRegression", "regime": "CALM", "horizon": 1, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "BTI_BritishAmerican_ret_20d", "FedFunds_zscore_60d", "EWL_Switzerland_zscore_60d", "MO_AltriaMG_ret_1d", "CCI_CrownCastle_vol_20d", "XLF_Fin_vol_20d", "ASX_Australia_vol_20d", "MSTR_Bitcoin3_ret_5d", "M_Macys_vol_20d", "XLV_Health_zscore_60d", "spx_abs_ret_max_5d", "T10Y2Y_Spread_ret_5d", "HD_ret_5d"], "is_new": true}, {"model_id": "new_h1_CALM_LogisticRegression_N15_t7", "algo": "LogisticRegression", "regime": "CALM", "horizon": 1, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "ORCL_zscore_60d", "AMD_ret_1d", "EWG_Germany_ret_20d", "spx_momentum_3d", "SPY_zscore_60d", "HD_ret_5d", "EWA_Australia_zscore_60d", "ASX_Australia_ret_5d", "WTI_Oil_FRED_zscore_60d", "SLB_Schlumberger_ret_5d", "EWM_Malaysia_zscore_60d", "XLV_Health_zscore_60d", "EOG_EOGResources_ret_5d"], "is_new": true}, {"model_id": "new_h1_CALM_LogisticRegression_N20_t0", "algo": "LogisticRegression", "regime": "CALM", "horizon": 1, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "spx_abs_ret_max_5d", "3M_ret_5d", "EFFR_vol_20d", "CLX_Clorox_vol_20d", "GILD_Gilead_ret_20d", "M_Macys_vol_20d", "LMT_LockheedMartin_vol_20d", "vix_mean_abs_ret_5d", "MS_MorganStanley_zscore_60d", "Michigan_Sentiment_ret_20d", "VOD_Vodafone_zscore_60d", "EWG_Germany_ret_20d", "SPY_zscore_60d", "AMD_ret_5d", "BTI_BritishAmerican_ret_5d", "CMCSA_ret_1d", "T_ret_1d", "HD_ret_20d"], "is_new": true}, {"model_id": "new_h1_CALM_LogisticRegression_N20_t1", "algo": "LogisticRegression", "regime": "CALM", "horizon": 1, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWM_Malaysia_ret_1d", "XLF_Fin_vol_20d", "EWA_Australia_zscore_60d", "PCAR_PaccarInc_ret_5d", "Michigan_Sentiment_ret_20d", "US5Y_Rate_ret_5d", "3M_vol_20d", "EXC_Exelon_zscore_60d", "EOG_EOGResources_vol_20d", "NEE_NextEra_ret_20d", "SLB_Schlumberger_ret_1d", "CPB_CampbellSoup_zscore_60d", "US30Y_Rate_ret_20d", "Brent_Oil_FRED_ret_20d", "SJM_JM_Smucker_ret_5d", "HUM_Humana_ret_5d", "JNJ_ret_1d", "Core_CPI_zscore_60d"], "is_new": true}, {"model_id": "new_h1_CALM_LogisticRegression_N20_t2", "algo": "LogisticRegression", "regime": "CALM", "horizon": 1, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "MSTR_Bitcoin3_ret_1d", "US3M_Rate_vol_20d", "SBUX_ret_5d", "EWS_Singapore_ret_5d", "Industrial_Production_zscore_60d", "PFE_ret_1d", "VOD_Vodafone_zscore_60d", "AORD_AUS_zscore_60d", "Core_CPI_zscore_60d", "EXC_Exelon_zscore_60d", "BTI_BritishAmerican_ret_5d", "MSTR_Bitcoin3_ret_20d", "heston_ev_h3", "EWL_Switzerland_zscore_60d", "EWM_Malaysia_vol_20d", "INTC_ret_1d", "NFCI_ret_5d", "LMT_LockheedMartin_vol_20d"], "is_new": true}, {"model_id": "new_h1_CALM_LogisticRegression_N20_t3", "algo": "LogisticRegression", "regime": "CALM", "horizon": 1, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "HD_ret_5d", "CMCSA_ret_1d", "M_Macys_vol_20d", "JNJ_ret_1d", "Brent_Oil_FRED_ret_5d", "US1Y_Rate_ret_20d", "EQIX_Equinix_ret_5d", "QQQ_vol_20d", "INTC_ret_1d", "LMT_LockheedMartin_vol_20d", "EWL_Switzerland_vol_20d", "US3Y_Rate_ret_5d", "GD_GeneralDynamics_zscore_60d", "EQR_Equity_ret_1d", "EWA_Australia_ret_1d", "SJM_JM_Smucker_ret_1d", "EWG_Germany_ret_20d", "EWQ_France_ret_20d"], "is_new": true}, {"model_id": "new_h1_CALM_LogisticRegression_N20_t4", "algo": "LogisticRegression", "regime": "CALM", "horizon": 1, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWM_Malaysia_vol_20d", "EWG_Germany_ret_20d", "EWY_Korea_ret_20d", "XLY_Disc_vol_20d", "LOW_Lowes_ret_20d", "EFFR_ret_1d", "CPB_CampbellSoup_zscore_60d", "Core_PCE_zscore_60d", "CCI_CrownCastle_vol_20d", "spx_vol_5d", "SCHW_Schwab_ret_5d", "XLV_Health_zscore_60d", "3M_vol_20d", "TM_Telephone_vol_20d", "AVB_AvalonBay_zscore_60d", "SBUX_ret_5d", "MRK_Merck_zscore_60d", "INTC_ret_1d"], "is_new": true}, {"model_id": "new_h1_CALM_LogisticRegression_N20_t5", "algo": "LogisticRegression", "regime": "CALM", "horizon": 1, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "SLB_Schlumberger_ret_5d", "HD_ret_1d", "MS_MorganStanley_zscore_60d", "EWL_Switzerland_zscore_60d", "US5Y_Rate_ret_5d", "DAX_Germany_zscore_60d", "BA_ret_1d", "TED_Spread_vol_20d", "NWL_Newell_ret_20d", "Retail_Sales_zscore_60d", "TM_Telephone_vol_20d", "ASX_Australia_ret_5d", "SJM_JM_Smucker_ret_5d", "LLY_zscore_60d", "MS_MorganStanley_ret_5d", "EOG_EOGResources_vol_20d", "PFE_ret_1d", "Core_PCE_zscore_60d"], "is_new": true}, {"model_id": "new_h1_CALM_LogisticRegression_N20_t6", "algo": "LogisticRegression", "regime": "CALM", "horizon": 1, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "AMD_ret_5d", "EWH_HongKong_ret_5d", "MSTR_Bitcoin3_ret_20d", "US6M_Rate_ret_20d", "CPB_CampbellSoup_ret_5d", "IWM_SmallCap_vol_20d", "LOW_Lowes_ret_5d", "XLK_Tech_zscore_60d", "EXC_Exelon_zscore_60d", "ES_Evergy_ret_1d", "T_ret_1d", "FedFunds_zscore_60d", "EWQ_France_ret_20d", "GD_GeneralDynamics_zscore_60d", "PPL_PPL_ret_1d", "JNJ_ret_1d", "LUV_SouthwestAir_ret_5d", "SJM_JM_Smucker_ret_1d"], "is_new": true}, {"model_id": "new_h1_CALM_LogisticRegression_N20_t7", "algo": "LogisticRegression", "regime": "CALM", "horizon": 1, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "LUV_SouthwestAir_ret_5d", "M_Macys_vol_20d", "HD_ret_1d", "PAYX_Paychex_zscore_60d", "EQR_Equity_ret_1d", "US3Y_Rate_ret_5d", "DE_Deere_ret_5d", "US5Y_Rate_ret_5d", "EWM_Malaysia_vol_20d", "CPB_CampbellSoup_ret_5d", "GE_ret_1d", "EWQ_France_ret_20d", "US1Y_Rate_ret_5d", "Nikkei_Japan_zscore_60d", "MSTR_Bitcoin3_ret_20d", "EWC_Canada_zscore_60d", "DIS_vol_20d", "DOW_Price_zscore_60d"], "is_new": true}, {"model_id": "new_h1_CALM_LogisticRegression_N25_t0", "algo": "LogisticRegression", "regime": "CALM", "horizon": 1, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "SJM_JM_Smucker_ret_1d", "DE_Deere_vol_20d", "gjr_condvar_h1", "SPY_zscore_60d", "TGT_Target_zscore_60d", "INTC_ret_1d", "SBUX_vol_20d", "HangSeng_HK_ret_1d", "EWY_Korea_zscore_60d", "US30Y_Rate_ret_20d", "US3M_Rate_zscore_60d", "AXP_Amex_vol_20d", "MS_MorganStanley_zscore_60d", "CPB_CampbellSoup_ret_5d", "EMR_Emerson_ret_20d", "BLK_BlackRock_zscore_60d", "EWL_Switzerland_vol_20d", "EXC_Exelon_ret_1d", "QQQ_vol_20d", "CPB_CampbellSoup_ret_20d", "PAYX_Paychex_zscore_60d", "IYR_US_REIT2_zscore_60d", "EXC_Exelon_zscore_60d"], "is_new": true}, {"model_id": "new_h1_CALM_LogisticRegression_N25_t1", "algo": "LogisticRegression", "regime": "CALM", "horizon": 1, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "AMT_AmericanTower_ret_1d", "NFCI_ret_5d", "DAX_Germany_vol_20d", "LMT_LockheedMartin_ret_1d", "XLF_Fin_vol_20d", "EWC_Canada_zscore_60d", "US5Y_Rate_ret_5d", "heston_var_ev_h7", "TM_Telephone_ret_1d", "EOG_EOGResources_vol_20d", "HD_ret_20d", "LMT_LockheedMartin_vol_20d", "IWM_SmallCap_vol_20d", "SLB_Schlumberger_ret_5d", "SBUX_vol_20d", "PAYX_Paychex_vol_20d", "CCI_CrownCastle_vol_20d", "MSTR_Bitcoin3_ret_20d", "HD_ret_5d", "PG_ret_20d", "DIS_vol_20d", "ENB_EnbridgeInc_ret_1d", "Industrial_Production_zscore_60d"], "is_new": true}, {"model_id": "new_h1_CALM_LogisticRegression_N25_t2", "algo": "LogisticRegression", "regime": "CALM", "horizon": 1, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EMR_Emerson_ret_20d", "JNJ_ret_1d", "EWM_Malaysia_vol_20d", "INTC_ret_1d", "INTC_ret_5d", "HD_zscore_60d", "GD_GeneralDynamics_zscore_60d", "XLB_Materials_zscore_60d", "MS_MorganStanley_zscore_60d", "GE_ret_1d", "PAYX_Paychex_ret_20d", "LOW_Lowes_ret_5d", "VOD_Vodafone_zscore_60d", "T10Y2Y_Spread_ret_5d", "PAYX_Paychex_zscore_60d", "NFCI_ret_5d", "ENB_EnbridgeInc_ret_1d", "ES_Evergy_ret_1d", "AMT_AmericanTower_ret_1d", "DAX_Germany_zscore_60d", "EWQ_France_ret_20d", "EWY_Korea_zscore_60d", "CCI_CrownCastle_vol_20d"], "is_new": true}, {"model_id": "new_h1_CALM_LogisticRegression_N25_t3", "algo": "LogisticRegression", "regime": "CALM", "horizon": 1, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "CPB_CampbellSoup_ret_20d", "BTI_BritishAmerican_ret_20d", "QQQ_vol_20d", "T_ret_1d", "BA_ret_1d", "AMZN_ret_5d", "MS_MorganStanley_zscore_60d", "TED_Spread_vol_20d", "CI_Cigna_vol_20d", "CPB_CampbellSoup_vol_20d", "FedFunds_zscore_60d", "XLY_Disc_vol_20d", "EFFR_ret_1d", "heston_var_ev_h3", "MS_MorganStanley_ret_1d", "IBEX_Spain_ret_20d", "EWM_Malaysia_vol_20d", "EWY_Korea_ret_20d", "HD_ret_20d", "PAYX_Paychex_ret_20d", "DAX_Germany_zscore_60d", "DE_Deere_ret_5d", "heston_var_ev_h5"], "is_new": true}, {"model_id": "new_h1_CALM_LogisticRegression_N25_t4", "algo": "LogisticRegression", "regime": "CALM", "horizon": 1, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "IWM_SmallCap_vol_20d", "QQQ_vol_20d", "PLD_Prologis_ret_5d", "TM_Telephone_vol_20d", "GE_ret_1d", "T_ret_1d", "PAYX_Paychex_ret_20d", "DE_Deere_vol_20d", "spx_momentum_3d", "EQIX_Equinix_ret_5d", "VRP_ma5", "VVIX_ret_20d", "PCAR_PaccarInc_ret_5d", "US3M_Rate_vol_20d", "ORCL_vol_20d", "TED_Spread_vol_20d", "TGT_Target_zscore_60d", "IBEX_Spain_ret_20d", "LLY_zscore_60d", "SBUX_ret_5d", "heston_var_ev_h3", "EWM_Malaysia_vol_20d", "CLX_Clorox_vol_20d"], "is_new": true}, {"model_id": "new_h1_CALM_LogisticRegression_N25_t5", "algo": "LogisticRegression", "regime": "CALM", "horizon": 1, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EXC_Exelon_zscore_60d", "EFFR_ret_1d", "HangSeng_HK_ret_5d", "AMD_ret_1d", "3M_ret_5d", "T10Y2Y_Spread_ret_5d", "DE_Deere_vol_20d", "LLY_zscore_60d", "HD_ret_5d", "PPL_PPL_ret_1d", "TGT_Target_zscore_60d", "spx_momentum_3d", "SBUX_zscore_60d", "Retail_Sales_zscore_60d", "HangSeng_HK_ret_1d", "XOM_ret_20d", "Nikkei_Japan_zscore_60d", "HD_zscore_60d", "VVIX_ret_20d", "EWM_Malaysia_zscore_60d", "3M_vol_20d", "T_ret_1d", "MRK_Merck_zscore_60d"], "is_new": true}, {"model_id": "new_h1_CALM_LogisticRegression_N25_t6", "algo": "LogisticRegression", "regime": "CALM", "horizon": 1, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWY_Korea_ret_20d", "EMR_Emerson_ret_20d", "heston_var_ev_h7", "MS_MorganStanley_ret_1d", "MSTR_Bitcoin3_ret_1d", "EWL_Switzerland_zscore_60d", "FedFunds_zscore_60d", "CCI_CrownCastle_vol_20d", "NWL_Newell_ret_20d", "XOM_ret_20d", "EWQ_France_zscore_60d", "SPY_zscore_60d", "XLK_Tech_zscore_60d", "GE_ret_1d", "HD_ret_20d", "TGT_Target_zscore_60d", "GD_GeneralDynamics_zscore_60d", "NFCI_ret_5d", "EWA_Australia_ret_1d", "BTI_BritishAmerican_ret_5d", "EQR_Equity_ret_1d", "DE_Deere_vol_20d", "HD_zscore_60d"], "is_new": true}, {"model_id": "new_h1_CALM_LogisticRegression_N25_t7", "algo": "LogisticRegression", "regime": "CALM", "horizon": 1, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "XOM_ret_1d", "SBUX_zscore_60d", "EWA_Australia_ret_1d", "ITT_ITTInc_ret_5d", "AMGN_Amgen_ret_1d", "DOW_Price_zscore_60d", "AXP_Amex_vol_20d", "TGT_Target_zscore_60d", "AMT_AmericanTower_ret_1d", "ES_Evergy_ret_1d", "CPB_CampbellSoup_vol_20d", "HD_ret_20d", "GD_GeneralDynamics_zscore_60d", "DAX_Germany_zscore_60d", "ORCL_zscore_60d", "AMD_ret_1d", "EWM_Malaysia_zscore_60d", "CMCSA_ret_1d", "hmm_p_stress", "AMD_ret_5d", "SCHW_Schwab_ret_5d", "EWL_Switzerland_vol_20d", "JNJ_ret_1d"], "is_new": true}, {"model_id": "new_h1_CALM_LogisticRegression_N30_t0", "algo": "LogisticRegression", "regime": "CALM", "horizon": 1, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "PG_ret_20d", "ASX_Australia_ret_5d", "IYM_BasicMaterials_ret_20d", "CCI_CrownCastle_vol_20d", "DIS_vol_20d", "BDX_Becton_Dickinson_ret_20d", "GD_GeneralDynamics_zscore_60d", "MS_MorganStanley_ret_5d", "Brent_Oil_FRED_ret_20d", "TXN_vol_20d", "PPL_PPL_ret_1d", "EQR_Equity_ret_1d", "EWA_Australia_zscore_60d", "EWY_Korea_zscore_60d", "MSTR_Bitcoin3_ret_20d", "AMD_ret_5d", "DAX_Germany_vol_20d", "EWY_Korea_ret_20d", "MSTR_Bitcoin3_ret_5d", "PAYX_Paychex_ret_20d", "XLB_Materials_zscore_60d", "vix_acceleration_1d", "EWC_Canada_zscore_60d", "PFE_ret_1d", "CPB_CampbellSoup_ret_20d", "T10Y2Y_Spread_ret_5d", "EWH_HongKong_ret_5d", "QQQ_vol_20d"], "is_new": true}, {"model_id": "new_h1_CALM_LogisticRegression_N30_t1", "algo": "LogisticRegression", "regime": "CALM", "horizon": 1, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "gjr_condvar_h1", "AVB_AvalonBay_zscore_60d", "T_ret_1d", "T10Y2Y_Spread_ret_5d", "DAX_Germany_vol_20d", "EWY_Korea_zscore_60d", "SBUX_ret_5d", "heston_var_ev_h3", "HangSeng_HK_ret_1d", "EWC_Canada_zscore_60d", "LOW_Lowes_ret_5d", "BA_ret_1d", "EWL_Switzerland_vol_20d", "CTAS_Cintas_vol_20d", "EWQ_France_ret_20d", "ORCL_vol_20d", "IWM_SmallCap_vol_20d", "AMD_ret_5d", "ENB_EnbridgeInc_ret_1d", "GILD_Gilead_ret_20d", "US7Y_Rate_ret_20d", "Core_PCE_zscore_60d", "MS_MorganStanley_zscore_60d", "EWS_Singapore_ret_5d", "AMD_ret_1d", "vix_acceleration_1d", "QQQ_vol_20d", "EFFR_ret_1d"], "is_new": true}, {"model_id": "new_h1_CALM_LogisticRegression_N30_t2", "algo": "LogisticRegression", "regime": "CALM", "horizon": 1, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "SJM_JM_Smucker_ret_1d", "CMCSA_ret_1d", "XLY_Disc_vol_20d", "Industrial_Production_zscore_60d", "XOM_ret_20d", "BLK_BlackRock_zscore_60d", "hmm_p_stress", "LOW_Lowes_ret_20d", "EFFR_ret_1d", "TM_Telephone_ret_1d", "CPB_CampbellSoup_zscore_60d", "DAX_Germany_zscore_60d", "SBUX_vol_20d", "EWM_Malaysia_zscore_60d", "INTC_ret_5d", "US1Y_Rate_ret_5d", "LLY_zscore_60d", "ORCL_zscore_60d", "EWA_Australia_zscore_60d", "AMT_AmericanTower_ret_1d", "MSTR_Bitcoin3_ret_1d", "heston_var_ev_h5", "DE_Deere_vol_20d", "EWY_Korea_zscore_60d", "AVB_AvalonBay_zscore_60d", "XOM_ret_1d", "XLB_Materials_zscore_60d", "US30Y_Rate_ret_20d"], "is_new": true}, {"model_id": "new_h1_CALM_LogisticRegression_N30_t3", "algo": "LogisticRegression", "regime": "CALM", "horizon": 1, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "TXN_vol_20d", "LMT_LockheedMartin_ret_1d", "ITT_ITTInc_ret_5d", "PLD_Prologis_ret_5d", "GILD_Gilead_ret_20d", "WTI_Oil_FRED_zscore_60d", "US1Y_Rate_ret_20d", "EMR_Emerson_ret_20d", "BTI_BritishAmerican_ret_5d", "TED_Spread_zscore_60d", "EFFR_ret_1d", "PAYX_Paychex_zscore_60d", "HUM_Humana_ret_5d", "FedFunds_zscore_60d", "TM_Telephone_vol_20d", "LUV_SouthwestAir_ret_5d", "PCAR_PaccarInc_ret_5d", "3M_ret_5d", "SLB_Schlumberger_ret_1d", "MSTR_Bitcoin3_ret_20d", "ES_Evergy_ret_1d", "AMGN_Amgen_ret_1d", "XLK_Tech_zscore_60d", "EWQ_France_zscore_60d", "spx_vol_5d", "BTI_BritishAmerican_ret_20d", "TM_Telephone_ret_1d", "MS_MorganStanley_ret_1d"], "is_new": true}, {"model_id": "new_h1_CALM_LogisticRegression_N30_t4", "algo": "LogisticRegression", "regime": "CALM", "horizon": 1, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "ITT_ITTInc_ret_5d", "NEE_NextEra_ret_20d", "heston_var_ev_h5", "vix_mean_abs_ret_5d", "CTAS_Cintas_vol_20d", "CPB_CampbellSoup_ret_20d", "T10Y2Y_Spread_ret_5d", "BTI_BritishAmerican_ret_5d", "AXP_Amex_ret_20d", "DE_Deere_ret_5d", "EQIX_Equinix_ret_5d", "ASX_Australia_ret_5d", "EWG_Germany_ret_20d", "CPB_CampbellSoup_zscore_60d", "SBUX_vol_20d", "EWA_Australia_zscore_60d", "CPB_CampbellSoup_ret_5d", "XOM_ret_20d", "US3Y_Rate_ret_5d", "MS_MorganStanley_ret_1d", "SBUX_zscore_60d", "TED_Spread_vol_20d", "EXC_Exelon_ret_1d", "AXP_Amex_vol_20d", "LUV_SouthwestAir_ret_5d", "EWL_Switzerland_zscore_60d", "Brent_Oil_FRED_ret_20d", "AMD_ret_5d"], "is_new": true}, {"model_id": "new_h1_CALM_LogisticRegression_N30_t5", "algo": "LogisticRegression", "regime": "CALM", "horizon": 1, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "heston_var_ev_h3", "BLK_BlackRock_zscore_60d", "LOW_Lowes_ret_5d", "US1Y_Rate_ret_5d", "NOC_Northrop_ret_20d", "MO_AltriaMG_ret_1d", "BDX_Becton_Dickinson_ret_20d", "XLV_Health_zscore_60d", "GD_GeneralDynamics_zscore_60d", "TED_Spread_zscore_60d", "AVB_AvalonBay_zscore_60d", "Brent_Oil_FRED_ret_20d", "HangSeng_HK_ret_5d", "Brent_Oil_FRED_ret_5d", "EWL_Switzerland_vol_20d", "SPY_zscore_60d", "CPB_CampbellSoup_ret_20d", "CPB_CampbellSoup_ret_5d", "EWM_Malaysia_zscore_60d", "TGT_Target_zscore_60d", "MSTR_Bitcoin3_ret_5d", "EWA_Australia_ret_1d", "EWA_Australia_zscore_60d", "TED_Spread_vol_20d", "EWY_Korea_ret_20d", "US3M_Rate_zscore_60d", "EQIX_Equinix_ret_5d", "spx_momentum_3d"], "is_new": true}, {"model_id": "new_h1_CALM_LogisticRegression_N30_t6", "algo": "LogisticRegression", "regime": "CALM", "horizon": 1, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "PFE_ret_1d", "vix_acceleration_1d", "Nikkei_Japan_vol_20d", "CPB_CampbellSoup_vol_20d", "ENB_EnbridgeInc_ret_1d", "EQR_Equity_ret_1d", "T_ret_1d", "M_Macys_vol_20d", "SLB_Schlumberger_ret_1d", "Industrial_Production_zscore_60d", "EFFR_vol_20d", "TGT_Target_zscore_60d", "WTI_Oil_FRED_zscore_60d", "PAYX_Paychex_ret_20d", "LMT_LockheedMartin_vol_20d", "INTC_ret_1d", "heston_var_ev_h7", "IWM_SmallCap_vol_20d", "MRK_Merck_zscore_60d", "NFCI_ret_5d", "Nikkei_Japan_zscore_60d", "ITT_ITTInc_ret_5d", "EWQ_France_zscore_60d", "US3M_Rate_vol_20d", "TXN_vol_20d", "EWJ_Japan_vol_20d", "MSTR_Bitcoin3_ret_1d", "XLF_Fin_vol_20d"], "is_new": true}, {"model_id": "new_h1_CALM_LogisticRegression_N30_t7", "algo": "LogisticRegression", "regime": "CALM", "horizon": 1, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "PAYX_Paychex_vol_20d", "MS_MorganStanley_zscore_60d", "3M_ret_5d", "ASX_Australia_ret_5d", "HD_ret_20d", "VVIX_ret_20d", "AMGN_Amgen_ret_1d", "SPY_zscore_60d", "DE_Deere_vol_20d", "HangSeng_HK_ret_5d", "3M_vol_20d", "HangSeng_HK_ret_1d", "EWL_Switzerland_zscore_60d", "EWJ_Japan_vol_20d", "INTC_ret_1d", "AMT_AmericanTower_ret_1d", "GILD_Gilead_ret_20d", "US1Y_Rate_ret_20d", "SLB_Schlumberger_ret_5d", "US1Y_Rate_ret_5d", "US3Y_Rate_ret_5d", "VRP_ma5", "EFFR_ret_1d", "CPB_CampbellSoup_ret_20d", "ASX_Australia_vol_20d", "PFE_ret_1d", "LOW_Lowes_ret_20d", "US7Y_Rate_ret_20d"], "is_new": true}, {"model_id": "new_h1_NORMAL_XGBoost_N5_t0", "algo": "XGBoost", "regime": "NORMAL", "horizon": 1, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWM_Malaysia_zscore_60d", "EXC_Exelon_ret_1d", "HangSeng_HK_ret_1d"], "is_new": true}, {"model_id": "new_h1_NORMAL_XGBoost_N5_t1", "algo": "XGBoost", "regime": "NORMAL", "horizon": 1, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "IWM_SmallCap_vol_20d", "SBUX_ret_5d", "EOG_EOGResources_vol_20d"], "is_new": true}, {"model_id": "new_h1_NORMAL_XGBoost_N5_t2", "algo": "XGBoost", "regime": "NORMAL", "horizon": 1, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "US5Y_Rate_ret_5d", "MSTR_Bitcoin3_ret_20d", "CCI_CrownCastle_vol_20d"], "is_new": true}, {"model_id": "new_h1_NORMAL_XGBoost_N5_t3", "algo": "XGBoost", "regime": "NORMAL", "horizon": 1, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "NOC_Northrop_ret_20d", "Core_PCE_zscore_60d", "EWM_Malaysia_vol_20d"], "is_new": true}, {"model_id": "new_h1_NORMAL_XGBoost_N5_t4", "algo": "XGBoost", "regime": "NORMAL", "horizon": 1, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "HangSeng_HK_vol_20d", "TED_Spread_vol_20d", "AVB_AvalonBay_zscore_60d"], "is_new": true}, {"model_id": "new_h1_NORMAL_XGBoost_N5_t5", "algo": "XGBoost", "regime": "NORMAL", "horizon": 1, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EFFR_ret_1d", "AXP_Amex_vol_20d", "MS_MorganStanley_ret_5d"], "is_new": true}, {"model_id": "new_h1_NORMAL_XGBoost_N5_t6", "algo": "XGBoost", "regime": "NORMAL", "horizon": 1, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "HD_zscore_60d", "MS_MorganStanley_ret_5d", "DIS_vol_20d"], "is_new": true}, {"model_id": "new_h1_NORMAL_XGBoost_N5_t7", "algo": "XGBoost", "regime": "NORMAL", "horizon": 1, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "US3Y_Rate_ret_5d", "AXP_Amex_vol_20d", "T_ret_1d"], "is_new": true}, {"model_id": "new_h1_NORMAL_XGBoost_N8_t0", "algo": "XGBoost", "regime": "NORMAL", "horizon": 1, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "SJM_JM_Smucker_ret_5d", "ORCL_vol_20d", "DE_Deere_ret_5d", "DE_Deere_vol_20d", "SBUX_ret_5d", "US30Y_Rate_ret_20d"], "is_new": true}, {"model_id": "new_h1_NORMAL_XGBoost_N8_t1", "algo": "XGBoost", "regime": "NORMAL", "horizon": 1, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "US30Y_Rate_ret_20d", "XLV_Health_zscore_60d", "TED_Spread_vol_20d", "EWY_Korea_ret_20d", "EWS_Singapore_ret_5d", "Brent_Oil_FRED_ret_20d"], "is_new": true}, {"model_id": "new_h1_NORMAL_XGBoost_N8_t2", "algo": "XGBoost", "regime": "NORMAL", "horizon": 1, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "CTAS_Cintas_vol_20d", "vix_mean_abs_ret_5d", "NFCI_ret_5d", "NWL_Newell_ret_20d", "LUV_SouthwestAir_ret_5d", "LMT_LockheedMartin_ret_1d"], "is_new": true}, {"model_id": "new_h1_NORMAL_XGBoost_N8_t3", "algo": "XGBoost", "regime": "NORMAL", "horizon": 1, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "HangSeng_HK_vol_20d", "EOG_EOGResources_ret_5d", "LMT_LockheedMartin_ret_1d", "XLV_Health_zscore_60d", "PAYX_Paychex_ret_20d", "XOM_ret_1d"], "is_new": true}, {"model_id": "new_h1_NORMAL_XGBoost_N8_t4", "algo": "XGBoost", "regime": "NORMAL", "horizon": 1, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWG_Germany_ret_20d", "MRK_Merck_zscore_60d", "EFFR_ret_1d", "CI_Cigna_vol_20d", "CTAS_Cintas_vol_20d", "BA_ret_1d"], "is_new": true}, {"model_id": "new_h1_NORMAL_XGBoost_N8_t5", "algo": "XGBoost", "regime": "NORMAL", "horizon": 1, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "JNJ_ret_1d", "CTAS_Cintas_vol_20d", "HangSeng_HK_vol_20d", "PAYX_Paychex_zscore_60d", "PPL_PPL_ret_1d", "AMT_AmericanTower_ret_1d"], "is_new": true}, {"model_id": "new_h1_NORMAL_XGBoost_N8_t6", "algo": "XGBoost", "regime": "NORMAL", "horizon": 1, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "TXN_vol_20d", "T10Y2Y_Spread_ret_5d", "EXC_Exelon_zscore_60d", "T_ret_1d", "SBUX_ret_5d", "XOM_ret_1d"], "is_new": true}, {"model_id": "new_h1_NORMAL_XGBoost_N8_t7", "algo": "XGBoost", "regime": "NORMAL", "horizon": 1, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "US7Y_Rate_ret_20d", "spx_vol_5d", "TED_Spread_vol_20d", "AXP_Amex_ret_20d", "EXC_Exelon_ret_1d", "EOG_EOGResources_ret_5d"], "is_new": true}, {"model_id": "new_h1_NORMAL_XGBoost_N10_t0", "algo": "XGBoost", "regime": "NORMAL", "horizon": 1, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "BA_ret_1d", "PLD_Prologis_ret_5d", "spx_abs_ret_max_5d", "AMD_ret_1d", "BTI_BritishAmerican_ret_5d", "MS_MorganStanley_ret_1d", "spx_vol_5d", "GE_ret_1d"], "is_new": true}, {"model_id": "new_h1_NORMAL_XGBoost_N10_t1", "algo": "XGBoost", "regime": "NORMAL", "horizon": 1, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "HD_zscore_60d", "hmm_p_stress", "MSTR_Bitcoin3_ret_5d", "EWM_Malaysia_zscore_60d", "MSTR_Bitcoin3_ret_20d", "SLB_Schlumberger_ret_5d", "CPB_CampbellSoup_ret_5d", "INTC_ret_1d"], "is_new": true}, {"model_id": "new_h1_NORMAL_XGBoost_N10_t2", "algo": "XGBoost", "regime": "NORMAL", "horizon": 1, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "SJM_JM_Smucker_ret_5d", "PFE_ret_1d", "US30Y_Rate_ret_20d", "AMT_AmericanTower_ret_1d", "AVB_AvalonBay_zscore_60d", "EWY_Korea_ret_20d", "TED_Spread_zscore_60d", "HangSeng_HK_vol_20d"], "is_new": true}, {"model_id": "new_h1_NORMAL_XGBoost_N10_t3", "algo": "XGBoost", "regime": "NORMAL", "horizon": 1, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "AMGN_Amgen_ret_1d", "LLY_zscore_60d", "LMT_LockheedMartin_vol_20d", "XOM_ret_20d", "HangSeng_HK_ret_5d", "EWQ_France_ret_20d", "SJM_JM_Smucker_ret_5d", "GILD_Gilead_ret_20d"], "is_new": true}, {"model_id": "new_h1_NORMAL_XGBoost_N10_t4", "algo": "XGBoost", "regime": "NORMAL", "horizon": 1, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "US1Y_Rate_ret_5d", "HD_ret_1d", "EWM_Malaysia_ret_1d", "DOW_Price_zscore_60d", "JNJ_ret_1d", "MS_MorganStanley_ret_5d", "spx_momentum_3d", "DHR_ret_1d"], "is_new": true}, {"model_id": "new_h1_NORMAL_XGBoost_N10_t5", "algo": "XGBoost", "regime": "NORMAL", "horizon": 1, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWQ_France_zscore_60d", "US3Y_Rate_ret_5d", "EFFR_ret_1d", "Nikkei_Japan_vol_20d", "IBEX_Spain_ret_20d", "XLB_Materials_zscore_60d", "IYM_BasicMaterials_ret_20d", "PAYX_Paychex_vol_20d"], "is_new": true}, {"model_id": "new_h1_NORMAL_XGBoost_N10_t6", "algo": "XGBoost", "regime": "NORMAL", "horizon": 1, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "US6M_Rate_ret_20d", "TM_Telephone_vol_20d", "EWG_Germany_ret_20d", "DHR_ret_1d", "IBEX_Spain_ret_20d", "TED_Spread_vol_20d", "DE_Deere_vol_20d", "Nikkei_Japan_zscore_60d"], "is_new": true}, {"model_id": "new_h1_NORMAL_XGBoost_N10_t7", "algo": "XGBoost", "regime": "NORMAL", "horizon": 1, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "US1Y_Rate_ret_5d", "TGT_Target_zscore_60d", "Core_PCE_zscore_60d", "ES_Evergy_ret_1d", "INTC_ret_5d", "DIS_vol_20d", "AORD_AUS_zscore_60d", "NFCI_ret_5d"], "is_new": true}, {"model_id": "new_h1_NORMAL_XGBoost_N12_t0", "algo": "XGBoost", "regime": "NORMAL", "horizon": 1, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "DOW_Price_zscore_60d", "EWA_Australia_ret_1d", "CPB_CampbellSoup_zscore_60d", "DE_Deere_ret_5d", "ES_Evergy_ret_1d", "AMZN_ret_5d", "XLK_Tech_zscore_60d", "XLV_Health_zscore_60d", "GD_GeneralDynamics_zscore_60d", "XLF_Fin_vol_20d"], "is_new": true}, {"model_id": "new_h1_NORMAL_XGBoost_N12_t1", "algo": "XGBoost", "regime": "NORMAL", "horizon": 1, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "AMZN_ret_5d", "LOW_Lowes_ret_20d", "CCI_CrownCastle_vol_20d", "Retail_Sales_zscore_60d", "DE_Deere_vol_20d", "SJM_JM_Smucker_ret_5d", "XLB_Materials_zscore_60d", "PPL_PPL_ret_1d", "hmm_p_stress", "MRK_Merck_zscore_60d"], "is_new": true}, {"model_id": "new_h1_NORMAL_XGBoost_N12_t2", "algo": "XGBoost", "regime": "NORMAL", "horizon": 1, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "DHR_ret_1d", "EMR_Emerson_ret_20d", "EWA_Australia_zscore_60d", "EWM_Malaysia_ret_1d", "ASX_Australia_ret_5d", "NEE_NextEra_ret_20d", "AMD_ret_5d", "GE_ret_1d", "NFCI_ret_5d", "EWL_Switzerland_zscore_60d"], "is_new": true}, {"model_id": "new_h1_NORMAL_XGBoost_N12_t3", "algo": "XGBoost", "regime": "NORMAL", "horizon": 1, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EQIX_Equinix_ret_5d", "ORCL_zscore_60d", "XLV_Health_zscore_60d", "SBUX_zscore_60d", "IWM_SmallCap_vol_20d", "PFE_ret_1d", "EWY_Korea_ret_20d", "Nikkei_Japan_vol_20d", "Michigan_Sentiment_ret_20d", "AVB_AvalonBay_zscore_60d"], "is_new": true}, {"model_id": "new_h1_NORMAL_XGBoost_N12_t4", "algo": "XGBoost", "regime": "NORMAL", "horizon": 1, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "XLK_Tech_zscore_60d", "US3Y_Rate_ret_5d", "VVIX_ret_20d", "TM_Telephone_ret_1d", "TED_Spread_vol_20d", "heston_var_ev_h5", "ASX_Australia_vol_20d", "T10Y2Y_Spread_ret_5d", "CPB_CampbellSoup_zscore_60d", "PAYX_Paychex_zscore_60d"], "is_new": true}, {"model_id": "new_h1_NORMAL_XGBoost_N12_t5", "algo": "XGBoost", "regime": "NORMAL", "horizon": 1, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "AVB_AvalonBay_zscore_60d", "US1Y_Rate_ret_5d", "MS_MorganStanley_ret_1d", "SPY_zscore_60d", "gjr_condvar_h1", "AXP_Amex_vol_20d", "EWA_Australia_ret_1d", "HD_zscore_60d", "ORCL_zscore_60d", "MS_MorganStanley_zscore_60d"], "is_new": true}, {"model_id": "new_h1_NORMAL_XGBoost_N12_t6", "algo": "XGBoost", "regime": "NORMAL", "horizon": 1, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "NWL_Newell_ret_20d", "HUM_Humana_ret_5d", "XLK_Tech_zscore_60d", "BA_ret_1d", "SPY_zscore_60d", "DHR_vol_20d", "PG_ret_20d", "heston_var_ev_h7", "PAYX_Paychex_zscore_60d", "CMCSA_ret_1d"], "is_new": true}, {"model_id": "new_h1_NORMAL_XGBoost_N12_t7", "algo": "XGBoost", "regime": "NORMAL", "horizon": 1, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "MS_MorganStanley_ret_5d", "NFCI_ret_5d", "ITT_ITTInc_ret_5d", "PAYX_Paychex_zscore_60d", "AMZN_ret_5d", "EQIX_Equinix_ret_5d", "PCAR_PaccarInc_ret_5d", "VRP_ma5", "EWG_Germany_ret_20d", "CMCSA_ret_1d"], "is_new": true}, {"model_id": "new_h1_NORMAL_XGBoost_N15_t0", "algo": "XGBoost", "regime": "NORMAL", "horizon": 1, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "CMCSA_ret_1d", "Core_CPI_zscore_60d", "PAYX_Paychex_zscore_60d", "heston_ev_h3", "LOW_Lowes_ret_5d", "CI_Cigna_vol_20d", "TED_Spread_vol_20d", "MSTR_Bitcoin3_ret_5d", "SLB_Schlumberger_ret_5d", "XLK_Tech_zscore_60d", "EFFR_vol_20d", "MSTR_Bitcoin3_ret_1d", "T_ret_1d"], "is_new": true}, {"model_id": "new_h1_NORMAL_XGBoost_N15_t1", "algo": "XGBoost", "regime": "NORMAL", "horizon": 1, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "US3M_Rate_vol_20d", "spx_momentum_3d", "DOW_Price_zscore_60d", "EMR_Emerson_ret_20d", "DIS_vol_20d", "EQIX_Equinix_ret_5d", "BA_ret_1d", "NVDA_vol_20d", "HangSeng_HK_vol_20d", "IBEX_Spain_ret_20d", "PAYX_Paychex_vol_20d", "HD_ret_5d", "XOM_ret_20d"], "is_new": true}, {"model_id": "new_h1_NORMAL_XGBoost_N15_t2", "algo": "XGBoost", "regime": "NORMAL", "horizon": 1, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "3M_ret_5d", "XOM_ret_1d", "EWY_Korea_zscore_60d", "AMD_ret_5d", "PFE_ret_1d", "SJM_JM_Smucker_ret_1d", "TXN_vol_20d", "3M_vol_20d", "XLF_Fin_vol_20d", "SCHW_Schwab_ret_5d", "EWQ_France_ret_20d", "EWL_Switzerland_vol_20d", "VOD_Vodafone_zscore_60d"], "is_new": true}, {"model_id": "new_h1_NORMAL_XGBoost_N15_t3", "algo": "XGBoost", "regime": "NORMAL", "horizon": 1, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "PAYX_Paychex_ret_20d", "CCI_CrownCastle_vol_20d", "WTI_Oil_FRED_zscore_60d", "3M_vol_20d", "EWS_Singapore_ret_5d", "T_ret_1d", "HD_ret_1d", "XLY_Disc_vol_20d", "XOM_ret_20d", "US1Y_Rate_ret_5d", "EWH_HongKong_ret_5d", "AXP_Amex_vol_20d", "ES_Evergy_ret_1d"], "is_new": true}, {"model_id": "new_h1_NORMAL_XGBoost_N15_t4", "algo": "XGBoost", "regime": "NORMAL", "horizon": 1, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "DIS_vol_20d", "ASX_Australia_ret_5d", "vix_acceleration_1d", "US5Y_Rate_ret_5d", "XLB_Materials_zscore_60d", "HangSeng_HK_ret_1d", "IYM_BasicMaterials_ret_20d", "CLX_Clorox_vol_20d", "AXP_Amex_vol_20d", "CPB_CampbellSoup_zscore_60d", "NOC_Northrop_ret_20d", "EQIX_Equinix_ret_5d", "EMR_Emerson_ret_20d"], "is_new": true}, {"model_id": "new_h1_NORMAL_XGBoost_N15_t5", "algo": "XGBoost", "regime": "NORMAL", "horizon": 1, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "US6M_Rate_ret_20d", "LOW_Lowes_ret_5d", "ASX_Australia_vol_20d", "CMCSA_ret_1d", "HangSeng_HK_ret_1d", "ITT_ITTInc_ret_5d", "MO_AltriaMG_ret_1d", "MSTR_Bitcoin3_ret_20d", "BTI_BritishAmerican_ret_5d", "Nikkei_Japan_vol_20d", "T_ret_1d", "BTI_BritishAmerican_ret_20d", "HD_zscore_60d"], "is_new": true}, {"model_id": "new_h1_NORMAL_XGBoost_N15_t6", "algo": "XGBoost", "regime": "NORMAL", "horizon": 1, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "IYM_BasicMaterials_ret_20d", "TED_Spread_vol_20d", "NFCI_ret_5d", "EWH_HongKong_ret_5d", "XLK_Tech_zscore_60d", "BDX_Becton_Dickinson_ret_20d", "CPB_CampbellSoup_zscore_60d", "US3Y_Rate_ret_5d", "gjr_condvar_h1", "TED_Spread_zscore_60d", "CLX_Clorox_vol_20d", "SBUX_ret_5d", "XLB_Materials_zscore_60d"], "is_new": true}, {"model_id": "new_h1_NORMAL_XGBoost_N15_t7", "algo": "XGBoost", "regime": "NORMAL", "horizon": 1, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EXC_Exelon_zscore_60d", "TED_Spread_vol_20d", "US5Y_Rate_ret_5d", "XOM_ret_1d", "LMT_LockheedMartin_vol_20d", "DE_Deere_ret_5d", "XLV_Health_zscore_60d", "US3M_Rate_vol_20d", "SJM_JM_Smucker_ret_1d", "T10Y2Y_Spread_ret_5d", "gjr_condvar_h1", "SCHW_Schwab_ret_5d", "EQR_Equity_ret_1d"], "is_new": true}, {"model_id": "new_h1_NORMAL_XGBoost_N20_t0", "algo": "XGBoost", "regime": "NORMAL", "horizon": 1, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "3M_ret_5d", "Core_PCE_zscore_60d", "AMZN_ret_5d", "PLD_Prologis_ret_5d", "XOM_ret_20d", "EWH_HongKong_ret_5d", "XLV_Health_zscore_60d", "NOC_Northrop_ret_20d", "NFCI_ret_5d", "AXP_Amex_ret_20d", "IYR_US_REIT2_zscore_60d", "LMT_LockheedMartin_ret_1d", "US30Y_Rate_ret_20d", "heston_var_ev_h3", "EWJ_Japan_vol_20d", "AMD_ret_1d", "EWG_Germany_ret_20d", "HD_ret_20d"], "is_new": true}, {"model_id": "new_h1_NORMAL_XGBoost_N20_t1", "algo": "XGBoost", "regime": "NORMAL", "horizon": 1, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "SLB_Schlumberger_ret_1d", "T_ret_1d", "ITT_ITTInc_ret_5d", "3M_vol_20d", "Core_CPI_zscore_60d", "IYR_US_REIT2_zscore_60d", "JNJ_ret_1d", "MS_MorganStanley_zscore_60d", "US3M_Rate_zscore_60d", "XLV_Health_zscore_60d", "AORD_AUS_zscore_60d", "EOG_EOGResources_vol_20d", "EWA_Australia_zscore_60d", "IYM_BasicMaterials_ret_20d", "Industrial_Production_zscore_60d", "Retail_Sales_zscore_60d", "QQQ_vol_20d", "ASX_Australia_vol_20d"], "is_new": true}, {"model_id": "new_h1_NORMAL_XGBoost_N20_t2", "algo": "XGBoost", "regime": "NORMAL", "horizon": 1, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "XOM_ret_20d", "PFE_ret_1d", "TM_Telephone_ret_1d", "IYR_US_REIT2_zscore_60d", "PG_ret_20d", "NFCI_ret_5d", "BDX_Becton_Dickinson_ret_20d", "T10Y2Y_Spread_ret_5d", "HangSeng_HK_ret_1d", "US3M_Rate_vol_20d", "INTC_ret_1d", "US1Y_Rate_ret_5d", "AMGN_Amgen_ret_1d", "EWH_HongKong_ret_5d", "US5Y_Rate_ret_5d", "IYM_BasicMaterials_ret_20d", "LMT_LockheedMartin_vol_20d", "BTI_BritishAmerican_ret_20d"], "is_new": true}, {"model_id": "new_h1_NORMAL_XGBoost_N20_t3", "algo": "XGBoost", "regime": "NORMAL", "horizon": 1, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "MSTR_Bitcoin3_ret_1d", "XOM_ret_20d", "TGT_Target_zscore_60d", "ENB_EnbridgeInc_ret_1d", "US3Y_Rate_ret_5d", "XLF_Fin_vol_20d", "MS_MorganStanley_ret_1d", "ES_Evergy_ret_1d", "VVIX_ret_20d", "Core_CPI_zscore_60d", "HangSeng_HK_vol_20d", "INTC_ret_1d", "ASX_Australia_ret_5d", "GE_ret_1d", "NFCI_ret_5d", "CLX_Clorox_vol_20d", "SJM_JM_Smucker_ret_1d", "EWQ_France_ret_20d"], "is_new": true}, {"model_id": "new_h1_NORMAL_XGBoost_N20_t4", "algo": "XGBoost", "regime": "NORMAL", "horizon": 1, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "TED_Spread_vol_20d", "HangSeng_HK_ret_5d", "EOG_EOGResources_vol_20d", "CTAS_Cintas_vol_20d", "MRK_Merck_zscore_60d", "ASX_Australia_vol_20d", "EWG_Germany_ret_20d", "CMCSA_ret_1d", "PAYX_Paychex_ret_20d", "SLB_Schlumberger_ret_1d", "AMZN_ret_5d", "SBUX_ret_5d", "SJM_JM_Smucker_ret_5d", "EFFR_ret_1d", "EWA_Australia_ret_1d", "JNJ_ret_1d", "IBEX_Spain_ret_20d", "IWM_SmallCap_vol_20d"], "is_new": true}, {"model_id": "new_h1_NORMAL_XGBoost_N20_t5", "algo": "XGBoost", "regime": "NORMAL", "horizon": 1, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "ASX_Australia_vol_20d", "PFE_ret_1d", "MS_MorganStanley_ret_1d", "T10Y2Y_Spread_ret_5d", "Core_CPI_zscore_60d", "3M_vol_20d", "IBEX_Spain_ret_20d", "AXP_Amex_ret_20d", "VVIX_ret_20d", "CLX_Clorox_vol_20d", "IYR_US_REIT2_zscore_60d", "PLD_Prologis_ret_5d", "CPB_CampbellSoup_vol_20d", "spx_momentum_3d", "AMT_AmericanTower_ret_1d", "US5Y_Rate_ret_5d", "heston_var_ev_h3", "EWH_HongKong_ret_5d"], "is_new": true}, {"model_id": "new_h1_NORMAL_XGBoost_N20_t6", "algo": "XGBoost", "regime": "NORMAL", "horizon": 1, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "GILD_Gilead_ret_20d", "TGT_Target_zscore_60d", "INTC_ret_5d", "ORCL_zscore_60d", "vix_mean_abs_ret_5d", "Core_PCE_zscore_60d", "GE_ret_1d", "DOW_Price_zscore_60d", "EFFR_vol_20d", "Core_CPI_zscore_60d", "AMD_ret_5d", "XOM_ret_1d", "US30Y_Rate_ret_20d", "SO_SouthernCo_ret_5d", "CPB_CampbellSoup_vol_20d", "EWG_Germany_vol_20d", "TM_Telephone_vol_20d", "LMT_LockheedMartin_ret_1d"], "is_new": true}, {"model_id": "new_h1_NORMAL_XGBoost_N20_t7", "algo": "XGBoost", "regime": "NORMAL", "horizon": 1, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "HD_ret_20d", "US3Y_Rate_ret_5d", "EWS_Singapore_ret_5d", "EOG_EOGResources_ret_5d", "Brent_Oil_FRED_ret_20d", "IBEX_Spain_ret_20d", "XLK_Tech_zscore_60d", "Industrial_Production_zscore_60d", "TM_Telephone_vol_20d", "PAYX_Paychex_ret_20d", "AXP_Amex_vol_20d", "DE_Deere_ret_5d", "AORD_AUS_zscore_60d", "PPL_PPL_ret_1d", "CPB_CampbellSoup_zscore_60d", "MSTR_Bitcoin3_ret_5d", "BA_ret_1d", "CLX_Clorox_vol_20d"], "is_new": true}, {"model_id": "new_h1_NORMAL_XGBoost_N25_t0", "algo": "XGBoost", "regime": "NORMAL", "horizon": 1, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "XLV_Health_zscore_60d", "LOW_Lowes_ret_20d", "IBEX_Spain_ret_20d", "IYR_US_REIT2_zscore_60d", "EWC_Canada_zscore_60d", "TGT_Target_zscore_60d", "CCI_CrownCastle_vol_20d", "PLD_Prologis_ret_5d", "US3M_Rate_zscore_60d", "DE_Deere_ret_5d", "CMCSA_ret_1d", "heston_var_ev_h5", "CPB_CampbellSoup_ret_20d", "PCAR_PaccarInc_ret_5d", "spx_vol_5d", "XOM_ret_20d", "PPL_PPL_ret_1d", "EWM_Malaysia_zscore_60d", "EWM_Malaysia_ret_1d", "DHR_vol_20d", "EXC_Exelon_zscore_60d", "LUV_SouthwestAir_ret_5d", "MS_MorganStanley_ret_5d"], "is_new": true}, {"model_id": "new_h1_NORMAL_XGBoost_N25_t1", "algo": "XGBoost", "regime": "NORMAL", "horizon": 1, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "SBUX_vol_20d", "AORD_AUS_zscore_60d", "BTI_BritishAmerican_ret_20d", "QQQ_vol_20d", "CTAS_Cintas_vol_20d", "LMT_LockheedMartin_ret_1d", "Retail_Sales_zscore_60d", "PG_ret_20d", "EOG_EOGResources_ret_5d", "HD_ret_20d", "hmm_p_stress", "XLB_Materials_zscore_60d", "US5Y_Rate_ret_5d", "DAX_Germany_zscore_60d", "SLB_Schlumberger_ret_5d", "CPB_CampbellSoup_vol_20d", "Michigan_Sentiment_ret_20d", "CPB_CampbellSoup_zscore_60d", "IWM_SmallCap_vol_20d", "US1Y_Rate_ret_5d", "Brent_Oil_FRED_ret_20d", "HangSeng_HK_ret_5d", "TGT_Target_zscore_60d"], "is_new": true}, {"model_id": "new_h1_NORMAL_XGBoost_N25_t2", "algo": "XGBoost", "regime": "NORMAL", "horizon": 1, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "XLY_Disc_vol_20d", "EWS_Singapore_ret_5d", "T_ret_1d", "EFFR_vol_20d", "SBUX_ret_5d", "MSTR_Bitcoin3_ret_5d", "BTI_BritishAmerican_ret_20d", "JNJ_ret_1d", "EWH_HongKong_ret_5d", "VRP_ma5", "Nikkei_Japan_zscore_60d", "ORCL_vol_20d", "US3M_Rate_zscore_60d", "heston_var_ev_h5", "TXN_vol_20d", "EWQ_France_ret_20d", "EOG_EOGResources_vol_20d", "Brent_Oil_FRED_ret_5d", "US3Y_Rate_ret_5d", "SBUX_vol_20d", "Core_CPI_zscore_60d", "PG_ret_20d", "IWM_SmallCap_vol_20d"], "is_new": true}, {"model_id": "new_h1_NORMAL_XGBoost_N25_t3", "algo": "XGBoost", "regime": "NORMAL", "horizon": 1, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "HangSeng_HK_vol_20d", "Michigan_Sentiment_ret_20d", "BTI_BritishAmerican_ret_5d", "WTI_Oil_FRED_zscore_60d", "CPB_CampbellSoup_zscore_60d", "VVIX_ret_20d", "MSTR_Bitcoin3_ret_20d", "IWM_SmallCap_vol_20d", "SCHW_Schwab_ret_5d", "BLK_BlackRock_zscore_60d", "CMCSA_ret_1d", "EWL_Switzerland_vol_20d", "DHR_ret_1d", "US6M_Rate_ret_20d", "3M_ret_5d", "DIS_vol_20d", "DOW_Price_zscore_60d", "PAYX_Paychex_zscore_60d", "TGT_Target_zscore_60d", "HD_zscore_60d", "MS_MorganStanley_ret_1d", "US5Y_Rate_ret_5d", "PAYX_Paychex_vol_20d"], "is_new": true}, {"model_id": "new_h1_NORMAL_XGBoost_N25_t4", "algo": "XGBoost", "regime": "NORMAL", "horizon": 1, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "US3M_Rate_zscore_60d", "HangSeng_HK_ret_5d", "ORCL_vol_20d", "LLY_zscore_60d", "CLX_Clorox_vol_20d", "US7Y_Rate_ret_20d", "AMD_ret_1d", "XLK_Tech_zscore_60d", "SLB_Schlumberger_ret_5d", "Michigan_Sentiment_ret_20d", "DAX_Germany_zscore_60d", "HUM_Humana_ret_5d", "BLK_BlackRock_zscore_60d", "MO_AltriaMG_ret_1d", "T_ret_1d", "BA_ret_1d", "3M_ret_5d", "US30Y_Rate_ret_20d", "LOW_Lowes_ret_20d", "CTAS_Cintas_vol_20d", "HD_ret_20d", "AORD_AUS_zscore_60d", "EWQ_France_ret_20d"], "is_new": true}, {"model_id": "new_h1_NORMAL_XGBoost_N25_t5", "algo": "XGBoost", "regime": "NORMAL", "horizon": 1, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "Michigan_Sentiment_ret_20d", "MSTR_Bitcoin3_ret_20d", "ORCL_vol_20d", "TXN_vol_20d", "EWJ_Japan_vol_20d", "EOG_EOGResources_vol_20d", "QQQ_vol_20d", "DHR_ret_1d", "EWA_Australia_ret_1d", "US1Y_Rate_ret_5d", "MSTR_Bitcoin3_ret_5d", "LOW_Lowes_ret_5d", "NOC_Northrop_ret_20d", "EFFR_ret_1d", "HD_ret_5d", "DAX_Germany_zscore_60d", "DE_Deere_vol_20d", "AMD_ret_1d", "Brent_Oil_FRED_ret_20d", "AMGN_Amgen_ret_1d", "DHR_vol_20d", "SBUX_vol_20d", "spx_vol_5d"], "is_new": true}, {"model_id": "new_h1_NORMAL_XGBoost_N25_t6", "algo": "XGBoost", "regime": "NORMAL", "horizon": 1, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "PCAR_PaccarInc_ret_5d", "XLB_Materials_zscore_60d", "NWL_Newell_ret_20d", "XOM_ret_1d", "EQR_Equity_ret_1d", "vix_mean_abs_ret_5d", "SLB_Schlumberger_ret_5d", "spx_vol_5d", "US1Y_Rate_ret_20d", "PAYX_Paychex_zscore_60d", "MSTR_Bitcoin3_ret_5d", "LMT_LockheedMartin_vol_20d", "EWM_Malaysia_ret_1d", "ES_Evergy_ret_1d", "heston_var_ev_h5", "vix_acceleration_1d", "3M_vol_20d", "US5Y_Rate_ret_5d", "SJM_JM_Smucker_ret_1d", "EWQ_France_ret_20d", "SJM_JM_Smucker_ret_5d", "GD_GeneralDynamics_zscore_60d", "NFCI_ret_5d"], "is_new": true}, {"model_id": "new_h1_NORMAL_XGBoost_N25_t7", "algo": "XGBoost", "regime": "NORMAL", "horizon": 1, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "US3M_Rate_zscore_60d", "SBUX_ret_5d", "MO_AltriaMG_ret_1d", "CTAS_Cintas_vol_20d", "US6M_Rate_ret_20d", "vix_acceleration_1d", "CCI_CrownCastle_vol_20d", "DHR_vol_20d", "TED_Spread_vol_20d", "HD_ret_1d", "PLD_Prologis_ret_5d", "SPY_zscore_60d", "heston_var_ev_h7", "T10Y2Y_Spread_ret_5d", "XLK_Tech_zscore_60d", "CLX_Clorox_vol_20d", "EWS_Singapore_ret_5d", "PPL_PPL_ret_1d", "Brent_Oil_FRED_ret_5d", "TM_Telephone_vol_20d", "spx_abs_ret_max_5d", "IBEX_Spain_ret_20d", "BA_ret_1d"], "is_new": true}, {"model_id": "new_h1_NORMAL_XGBoost_N30_t0", "algo": "XGBoost", "regime": "NORMAL", "horizon": 1, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "SO_SouthernCo_ret_5d", "PPL_PPL_ret_1d", "US3Y_Rate_ret_5d", "IYM_BasicMaterials_ret_20d", "NVDA_vol_20d", "LMT_LockheedMartin_vol_20d", "HUM_Humana_ret_5d", "HD_zscore_60d", "ENB_EnbridgeInc_ret_1d", "ASX_Australia_vol_20d", "VVIX_ret_20d", "HangSeng_HK_ret_5d", "US3M_Rate_vol_20d", "EXC_Exelon_zscore_60d", "BLK_BlackRock_zscore_60d", "QQQ_vol_20d", "US6M_Rate_ret_20d", "EOG_EOGResources_ret_5d", "spx_vol_5d", "BA_ret_1d", "heston_var_ev_h5", "AMZN_ret_5d", "HD_ret_5d", "SBUX_vol_20d", "CI_Cigna_vol_20d", "DAX_Germany_zscore_60d", "SLB_Schlumberger_ret_1d", "Brent_Oil_FRED_ret_5d"], "is_new": true}, {"model_id": "new_h1_NORMAL_XGBoost_N30_t1", "algo": "XGBoost", "regime": "NORMAL", "horizon": 1, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "GILD_Gilead_ret_20d", "EXC_Exelon_ret_1d", "AMD_ret_5d", "CTAS_Cintas_vol_20d", "XLB_Materials_zscore_60d", "EOG_EOGResources_vol_20d", "VRP_ma5", "DOW_Price_zscore_60d", "PAYX_Paychex_vol_20d", "MO_AltriaMG_ret_1d", "hmm_p_stress", "EXC_Exelon_zscore_60d", "EWQ_France_ret_20d", "SBUX_vol_20d", "EQR_Equity_ret_1d", "TED_Spread_zscore_60d", "ASX_Australia_ret_5d", "JNJ_ret_1d", "CI_Cigna_vol_20d", "DIS_vol_20d", "EFFR_ret_1d", "SCHW_Schwab_ret_5d", "NFCI_ret_5d", "MSTR_Bitcoin3_ret_5d", "XOM_ret_1d", "ORCL_zscore_60d", "ENB_EnbridgeInc_ret_1d", "EWH_HongKong_ret_5d"], "is_new": true}, {"model_id": "new_h1_NORMAL_XGBoost_N30_t2", "algo": "XGBoost", "regime": "NORMAL", "horizon": 1, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "Core_CPI_zscore_60d", "BA_ret_1d", "ENB_EnbridgeInc_ret_1d", "PG_ret_20d", "M_Macys_vol_20d", "QQQ_vol_20d", "HangSeng_HK_ret_5d", "hmm_p_stress", "TGT_Target_zscore_60d", "EWM_Malaysia_zscore_60d", "MSTR_Bitcoin3_ret_20d", "MS_MorganStanley_ret_1d", "heston_var_ev_h5", "FedFunds_zscore_60d", "DIS_vol_20d", "TED_Spread_zscore_60d", "ORCL_vol_20d", "AMD_ret_5d", "HD_ret_5d", "PAYX_Paychex_vol_20d", "gjr_condvar_h1", "INTC_ret_1d", "XLK_Tech_zscore_60d", "AVB_AvalonBay_zscore_60d", "CPB_CampbellSoup_ret_20d", "heston_var_ev_h3", "LMT_LockheedMartin_vol_20d", "Nikkei_Japan_zscore_60d"], "is_new": true}, {"model_id": "new_h1_NORMAL_XGBoost_N30_t3", "algo": "XGBoost", "regime": "NORMAL", "horizon": 1, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "GD_GeneralDynamics_zscore_60d", "CPB_CampbellSoup_ret_5d", "EXC_Exelon_ret_1d", "spx_abs_ret_max_5d", "US7Y_Rate_ret_20d", "MS_MorganStanley_ret_5d", "PG_ret_20d", "EWQ_France_zscore_60d", "VOD_Vodafone_zscore_60d", "EWA_Australia_zscore_60d", "T10Y2Y_Spread_ret_5d", "Industrial_Production_zscore_60d", "EQR_Equity_ret_1d", "INTC_ret_1d", "T_ret_1d", "AMD_ret_1d", "XLK_Tech_zscore_60d", "BTI_BritishAmerican_ret_5d", "AMD_ret_5d", "NFCI_ret_5d", "vix_acceleration_1d", "TED_Spread_vol_20d", "CPB_CampbellSoup_vol_20d", "PAYX_Paychex_vol_20d", "LMT_LockheedMartin_ret_1d", "EWL_Switzerland_zscore_60d", "HD_ret_20d", "EWY_Korea_zscore_60d"], "is_new": true}, {"model_id": "new_h1_NORMAL_XGBoost_N30_t4", "algo": "XGBoost", "regime": "NORMAL", "horizon": 1, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "BDX_Becton_Dickinson_ret_20d", "VVIX_ret_20d", "spx_momentum_3d", "EWL_Switzerland_vol_20d", "WTI_Oil_FRED_zscore_60d", "heston_var_ev_h3", "DAX_Germany_vol_20d", "Core_PCE_zscore_60d", "CLX_Clorox_vol_20d", "XLB_Materials_zscore_60d", "DE_Deere_ret_5d", "MSTR_Bitcoin3_ret_1d", "US3M_Rate_vol_20d", "ES_Evergy_ret_1d", "CTAS_Cintas_vol_20d", "XLY_Disc_vol_20d", "EWY_Korea_zscore_60d", "HUM_Humana_ret_5d", "PLD_Prologis_ret_5d", "Brent_Oil_FRED_ret_5d", "PCAR_PaccarInc_ret_5d", "GILD_Gilead_ret_20d", "Nikkei_Japan_vol_20d", "ITT_ITTInc_ret_5d", "EXC_Exelon_ret_1d", "EMR_Emerson_ret_20d", "Brent_Oil_FRED_ret_20d", "VOD_Vodafone_zscore_60d"], "is_new": true}, {"model_id": "new_h1_NORMAL_XGBoost_N30_t5", "algo": "XGBoost", "regime": "NORMAL", "horizon": 1, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "BLK_BlackRock_zscore_60d", "EXC_Exelon_zscore_60d", "FedFunds_zscore_60d", "EQIX_Equinix_ret_5d", "ORCL_zscore_60d", "BTI_BritishAmerican_ret_5d", "SBUX_zscore_60d", "EWG_Germany_ret_20d", "XLK_Tech_zscore_60d", "MS_MorganStanley_ret_5d", "spx_momentum_3d", "heston_var_ev_h7", "CPB_CampbellSoup_zscore_60d", "DHR_vol_20d", "TM_Telephone_vol_20d", "Nikkei_Japan_zscore_60d", "EWL_Switzerland_vol_20d", "XOM_ret_1d", "PAYX_Paychex_ret_20d", "vix_acceleration_1d", "CPB_CampbellSoup_vol_20d", "ASX_Australia_ret_5d", "Brent_Oil_FRED_ret_20d", "US7Y_Rate_ret_20d", "US3Y_Rate_ret_5d", "EWY_Korea_zscore_60d", "PAYX_Paychex_vol_20d", "AMGN_Amgen_ret_1d"], "is_new": true}, {"model_id": "new_h1_NORMAL_XGBoost_N30_t6", "algo": "XGBoost", "regime": "NORMAL", "horizon": 1, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "vix_acceleration_1d", "MS_MorganStanley_ret_1d", "GD_GeneralDynamics_zscore_60d", "DOW_Price_zscore_60d", "IYM_BasicMaterials_ret_20d", "MS_MorganStanley_ret_5d", "US5Y_Rate_ret_5d", "3M_ret_5d", "SO_SouthernCo_ret_5d", "SBUX_zscore_60d", "Retail_Sales_zscore_60d", "ORCL_vol_20d", "Michigan_Sentiment_ret_20d", "BTI_BritishAmerican_ret_5d", "TM_Telephone_vol_20d", "EWM_Malaysia_ret_1d", "heston_var_ev_h5", "MSTR_Bitcoin3_ret_5d", "SBUX_ret_5d", "DAX_Germany_vol_20d", "US3M_Rate_zscore_60d", "TM_Telephone_ret_1d", "LOW_Lowes_ret_20d", "IYR_US_REIT2_zscore_60d", "HangSeng_HK_ret_5d", "M_Macys_vol_20d", "BDX_Becton_Dickinson_ret_20d", "CMCSA_ret_1d"], "is_new": true}, {"model_id": "new_h1_NORMAL_XGBoost_N30_t7", "algo": "XGBoost", "regime": "NORMAL", "horizon": 1, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "HD_zscore_60d", "heston_var_ev_h7", "EWG_Germany_ret_20d", "EQIX_Equinix_ret_5d", "US1Y_Rate_ret_20d", "AVB_AvalonBay_zscore_60d", "MSTR_Bitcoin3_ret_5d", "EWA_Australia_ret_1d", "HangSeng_HK_ret_1d", "MS_MorganStanley_ret_1d", "CPB_CampbellSoup_zscore_60d", "LOW_Lowes_ret_20d", "SBUX_vol_20d", "CI_Cigna_vol_20d", "ORCL_zscore_60d", "heston_var_ev_h5", "DHR_ret_1d", "ASX_Australia_vol_20d", "EWA_Australia_zscore_60d", "AMZN_ret_5d", "VRP_ma5", "EWQ_France_zscore_60d", "SBUX_zscore_60d", "heston_ev_h3", "BTI_BritishAmerican_ret_20d", "TM_Telephone_vol_20d", "HangSeng_HK_vol_20d", "SJM_JM_Smucker_ret_1d"], "is_new": true}, {"model_id": "new_h1_NORMAL_LightGBM_N5_t0", "algo": "LightGBM", "regime": "NORMAL", "horizon": 1, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "Industrial_Production_zscore_60d", "BDX_Becton_Dickinson_ret_20d", "AVB_AvalonBay_zscore_60d"], "is_new": true}, {"model_id": "new_h1_NORMAL_LightGBM_N5_t1", "algo": "LightGBM", "regime": "NORMAL", "horizon": 1, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "TM_Telephone_ret_1d", "DOW_Price_zscore_60d", "US6M_Rate_ret_20d"], "is_new": true}, {"model_id": "new_h1_NORMAL_LightGBM_N5_t2", "algo": "LightGBM", "regime": "NORMAL", "horizon": 1, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "SBUX_zscore_60d", "spx_abs_ret_max_5d", "AMGN_Amgen_ret_1d"], "is_new": true}, {"model_id": "new_h1_NORMAL_LightGBM_N5_t3", "algo": "LightGBM", "regime": "NORMAL", "horizon": 1, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "AMD_ret_5d", "EOG_EOGResources_vol_20d", "gjr_condvar_h1"], "is_new": true}, {"model_id": "new_h1_NORMAL_LightGBM_N5_t4", "algo": "LightGBM", "regime": "NORMAL", "horizon": 1, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "XLK_Tech_zscore_60d", "EOG_EOGResources_ret_5d", "VRP_ma5"], "is_new": true}, {"model_id": "new_h1_NORMAL_LightGBM_N5_t5", "algo": "LightGBM", "regime": "NORMAL", "horizon": 1, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "VVIX_ret_20d", "XLV_Health_zscore_60d", "EWL_Switzerland_zscore_60d"], "is_new": true}, {"model_id": "new_h1_NORMAL_LightGBM_N5_t6", "algo": "LightGBM", "regime": "NORMAL", "horizon": 1, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "M_Macys_vol_20d", "IWM_SmallCap_vol_20d", "Core_PCE_zscore_60d"], "is_new": true}, {"model_id": "new_h1_NORMAL_LightGBM_N5_t7", "algo": "LightGBM", "regime": "NORMAL", "horizon": 1, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "AXP_Amex_ret_20d", "VRP_ma5", "EOG_EOGResources_vol_20d"], "is_new": true}, {"model_id": "new_h1_NORMAL_LightGBM_N8_t0", "algo": "LightGBM", "regime": "NORMAL", "horizon": 1, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "DHR_vol_20d", "NOC_Northrop_ret_20d", "EOG_EOGResources_ret_5d", "MO_AltriaMG_ret_1d", "US1Y_Rate_ret_20d", "ENB_EnbridgeInc_ret_1d"], "is_new": true}, {"model_id": "new_h1_NORMAL_LightGBM_N8_t1", "algo": "LightGBM", "regime": "NORMAL", "horizon": 1, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "CPB_CampbellSoup_ret_5d", "gjr_condvar_h1", "XLK_Tech_zscore_60d", "MSTR_Bitcoin3_ret_1d", "EFFR_vol_20d", "AMGN_Amgen_ret_1d"], "is_new": true}, {"model_id": "new_h1_NORMAL_LightGBM_N8_t2", "algo": "LightGBM", "regime": "NORMAL", "horizon": 1, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EQIX_Equinix_ret_5d", "CLX_Clorox_vol_20d", "hmm_p_stress", "VRP_ma5", "HD_ret_20d", "NOC_Northrop_ret_20d"], "is_new": true}, {"model_id": "new_h1_NORMAL_LightGBM_N8_t3", "algo": "LightGBM", "regime": "NORMAL", "horizon": 1, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "DHR_vol_20d", "heston_var_ev_h3", "GE_ret_1d", "T10Y2Y_Spread_ret_5d", "PAYX_Paychex_zscore_60d", "NFCI_ret_5d"], "is_new": true}, {"model_id": "new_h1_NORMAL_LightGBM_N8_t4", "algo": "LightGBM", "regime": "NORMAL", "horizon": 1, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "SJM_JM_Smucker_ret_1d", "XLK_Tech_zscore_60d", "CLX_Clorox_vol_20d", "XOM_ret_20d", "HangSeng_HK_ret_5d", "LMT_LockheedMartin_ret_1d"], "is_new": true}, {"model_id": "new_h1_NORMAL_LightGBM_N8_t5", "algo": "LightGBM", "regime": "NORMAL", "horizon": 1, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "Core_PCE_zscore_60d", "TED_Spread_zscore_60d", "CPB_CampbellSoup_ret_20d", "XLY_Disc_vol_20d", "LOW_Lowes_ret_5d", "TM_Telephone_vol_20d"], "is_new": true}, {"model_id": "new_h1_NORMAL_LightGBM_N8_t6", "algo": "LightGBM", "regime": "NORMAL", "horizon": 1, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "GILD_Gilead_ret_20d", "BA_ret_1d", "EWA_Australia_zscore_60d", "CTAS_Cintas_vol_20d", "DOW_Price_zscore_60d", "QQQ_vol_20d"], "is_new": true}, {"model_id": "new_h1_NORMAL_LightGBM_N8_t7", "algo": "LightGBM", "regime": "NORMAL", "horizon": 1, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "BA_ret_1d", "heston_var_ev_h7", "LUV_SouthwestAir_ret_5d", "HD_ret_20d", "EWH_HongKong_ret_5d", "ORCL_vol_20d"], "is_new": true}, {"model_id": "new_h1_NORMAL_LightGBM_N10_t0", "algo": "LightGBM", "regime": "NORMAL", "horizon": 1, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "CPB_CampbellSoup_vol_20d", "BA_ret_1d", "TM_Telephone_ret_1d", "EQIX_Equinix_ret_5d", "EFFR_vol_20d", "AORD_AUS_zscore_60d", "SBUX_vol_20d", "SO_SouthernCo_ret_5d"], "is_new": true}, {"model_id": "new_h1_NORMAL_LightGBM_N10_t1", "algo": "LightGBM", "regime": "NORMAL", "horizon": 1, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "VRP_ma5", "US6M_Rate_ret_20d", "IWM_SmallCap_vol_20d", "NVDA_vol_20d", "EWC_Canada_zscore_60d", "EWM_Malaysia_vol_20d", "CTAS_Cintas_vol_20d", "CPB_CampbellSoup_ret_5d"], "is_new": true}, {"model_id": "new_h1_NORMAL_LightGBM_N10_t2", "algo": "LightGBM", "regime": "NORMAL", "horizon": 1, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWL_Switzerland_zscore_60d", "Brent_Oil_FRED_ret_20d", "PPL_PPL_ret_1d", "US7Y_Rate_ret_20d", "SBUX_vol_20d", "BA_ret_1d", "ORCL_vol_20d", "DIS_vol_20d"], "is_new": true}, {"model_id": "new_h1_NORMAL_LightGBM_N10_t3", "algo": "LightGBM", "regime": "NORMAL", "horizon": 1, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "MRK_Merck_zscore_60d", "DE_Deere_vol_20d", "PFE_ret_1d", "EWA_Australia_zscore_60d", "gjr_condvar_h1", "LOW_Lowes_ret_20d", "GE_ret_1d", "EWG_Germany_ret_20d"], "is_new": true}, {"model_id": "new_h1_NORMAL_LightGBM_N10_t4", "algo": "LightGBM", "regime": "NORMAL", "horizon": 1, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "HD_ret_1d", "DE_Deere_vol_20d", "CTAS_Cintas_vol_20d", "PLD_Prologis_ret_5d", "PFE_ret_1d", "NVDA_vol_20d", "EOG_EOGResources_ret_5d", "JNJ_ret_1d"], "is_new": true}, {"model_id": "new_h1_NORMAL_LightGBM_N10_t5", "algo": "LightGBM", "regime": "NORMAL", "horizon": 1, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "US5Y_Rate_ret_5d", "US1Y_Rate_ret_20d", "EWM_Malaysia_vol_20d", "CLX_Clorox_vol_20d", "NFCI_ret_5d", "T10Y2Y_Spread_ret_5d", "SBUX_zscore_60d", "EWL_Switzerland_vol_20d"], "is_new": true}, {"model_id": "new_h1_NORMAL_LightGBM_N10_t6", "algo": "LightGBM", "regime": "NORMAL", "horizon": 1, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "AMD_ret_5d", "Nikkei_Japan_vol_20d", "EWY_Korea_ret_20d", "IWM_SmallCap_vol_20d", "ORCL_vol_20d", "US6M_Rate_ret_20d", "EWL_Switzerland_zscore_60d", "VRP_ma5"], "is_new": true}, {"model_id": "new_h1_NORMAL_LightGBM_N10_t7", "algo": "LightGBM", "regime": "NORMAL", "horizon": 1, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "AVB_AvalonBay_zscore_60d", "PPL_PPL_ret_1d", "SLB_Schlumberger_ret_1d", "ASX_Australia_ret_5d", "NWL_Newell_ret_20d", "TED_Spread_zscore_60d", "MS_MorganStanley_ret_5d", "Retail_Sales_zscore_60d"], "is_new": true}, {"model_id": "new_h1_NORMAL_LightGBM_N12_t0", "algo": "LightGBM", "regime": "NORMAL", "horizon": 1, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWQ_France_zscore_60d", "CI_Cigna_vol_20d", "US1Y_Rate_ret_5d", "AMD_ret_5d", "NEE_NextEra_ret_20d", "EWL_Switzerland_vol_20d", "DE_Deere_ret_5d", "LOW_Lowes_ret_20d", "HUM_Humana_ret_5d", "heston_var_ev_h7"], "is_new": true}, {"model_id": "new_h1_NORMAL_LightGBM_N12_t1", "algo": "LightGBM", "regime": "NORMAL", "horizon": 1, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EMR_Emerson_ret_20d", "PFE_ret_1d", "US3Y_Rate_ret_5d", "US7Y_Rate_ret_20d", "AMZN_ret_5d", "PCAR_PaccarInc_ret_5d", "PG_ret_20d", "DHR_ret_1d", "Michigan_Sentiment_ret_20d", "ES_Evergy_ret_1d"], "is_new": true}, {"model_id": "new_h1_NORMAL_LightGBM_N12_t2", "algo": "LightGBM", "regime": "NORMAL", "horizon": 1, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "MSTR_Bitcoin3_ret_5d", "IYR_US_REIT2_zscore_60d", "TED_Spread_zscore_60d", "ASX_Australia_ret_5d", "HUM_Humana_ret_5d", "Nikkei_Japan_vol_20d", "GE_ret_1d", "XLK_Tech_zscore_60d", "TM_Telephone_vol_20d", "US5Y_Rate_ret_5d"], "is_new": true}, {"model_id": "new_h1_NORMAL_LightGBM_N12_t3", "algo": "LightGBM", "regime": "NORMAL", "horizon": 1, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "SO_SouthernCo_ret_5d", "LUV_SouthwestAir_ret_5d", "BA_ret_1d", "CTAS_Cintas_vol_20d", "MSTR_Bitcoin3_ret_5d", "PLD_Prologis_ret_5d", "XLY_Disc_vol_20d", "AVB_AvalonBay_zscore_60d", "EOG_EOGResources_vol_20d", "heston_var_ev_h7"], "is_new": true}, {"model_id": "new_h1_NORMAL_LightGBM_N12_t4", "algo": "LightGBM", "regime": "NORMAL", "horizon": 1, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "IYR_US_REIT2_zscore_60d", "EWQ_France_zscore_60d", "TM_Telephone_vol_20d", "Core_CPI_zscore_60d", "hmm_p_stress", "Retail_Sales_zscore_60d", "US6M_Rate_ret_20d", "EWH_HongKong_ret_5d", "EWM_Malaysia_ret_1d", "ITT_ITTInc_ret_5d"], "is_new": true}, {"model_id": "new_h1_NORMAL_LightGBM_N12_t5", "algo": "LightGBM", "regime": "NORMAL", "horizon": 1, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "BA_ret_1d", "BTI_BritishAmerican_ret_20d", "MRK_Merck_zscore_60d", "LLY_zscore_60d", "PAYX_Paychex_zscore_60d", "SBUX_vol_20d", "SJM_JM_Smucker_ret_5d", "TED_Spread_vol_20d", "DAX_Germany_vol_20d", "PPL_PPL_ret_1d"], "is_new": true}, {"model_id": "new_h1_NORMAL_LightGBM_N12_t6", "algo": "LightGBM", "regime": "NORMAL", "horizon": 1, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "BTI_BritishAmerican_ret_20d", "EWA_Australia_ret_1d", "PFE_ret_1d", "CLX_Clorox_vol_20d", "CCI_CrownCastle_vol_20d", "MS_MorganStanley_ret_5d", "EQIX_Equinix_ret_5d", "HD_ret_5d", "Brent_Oil_FRED_ret_5d", "CPB_CampbellSoup_zscore_60d"], "is_new": true}, {"model_id": "new_h1_NORMAL_LightGBM_N12_t7", "algo": "LightGBM", "regime": "NORMAL", "horizon": 1, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "SJM_JM_Smucker_ret_1d", "DOW_Price_zscore_60d", "EWG_Germany_vol_20d", "T10Y2Y_Spread_ret_5d", "XOM_ret_20d", "AMT_AmericanTower_ret_1d", "HUM_Humana_ret_5d", "NOC_Northrop_ret_20d", "TM_Telephone_vol_20d", "DE_Deere_ret_5d"], "is_new": true}, {"model_id": "new_h1_NORMAL_LightGBM_N15_t0", "algo": "LightGBM", "regime": "NORMAL", "horizon": 1, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "heston_var_ev_h7", "AMT_AmericanTower_ret_1d", "HD_zscore_60d", "EWA_Australia_ret_1d", "BTI_BritishAmerican_ret_20d", "EWM_Malaysia_zscore_60d", "T10Y2Y_Spread_ret_5d", "IYR_US_REIT2_zscore_60d", "NFCI_ret_5d", "US6M_Rate_ret_20d", "JNJ_ret_1d", "spx_abs_ret_max_5d", "IWM_SmallCap_vol_20d"], "is_new": true}, {"model_id": "new_h1_NORMAL_LightGBM_N15_t1", "algo": "LightGBM", "regime": "NORMAL", "horizon": 1, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "US3M_Rate_zscore_60d", "DE_Deere_ret_5d", "HangSeng_HK_vol_20d", "EOG_EOGResources_ret_5d", "EXC_Exelon_zscore_60d", "EWM_Malaysia_vol_20d", "SBUX_ret_5d", "MRK_Merck_zscore_60d", "HD_ret_5d", "AXP_Amex_vol_20d", "DIS_vol_20d", "LMT_LockheedMartin_vol_20d", "HD_ret_20d"], "is_new": true}, {"model_id": "new_h1_NORMAL_LightGBM_N15_t2", "algo": "LightGBM", "regime": "NORMAL", "horizon": 1, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EQR_Equity_ret_1d", "EOG_EOGResources_vol_20d", "vix_acceleration_1d", "EFFR_ret_1d", "EMR_Emerson_ret_20d", "EWQ_France_ret_20d", "Core_CPI_zscore_60d", "AVB_AvalonBay_zscore_60d", "INTC_ret_5d", "US5Y_Rate_ret_5d", "DAX_Germany_zscore_60d", "ORCL_zscore_60d", "EXC_Exelon_zscore_60d"], "is_new": true}, {"model_id": "new_h1_NORMAL_LightGBM_N15_t3", "algo": "LightGBM", "regime": "NORMAL", "horizon": 1, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "vix_acceleration_1d", "INTC_ret_5d", "LLY_zscore_60d", "DIS_vol_20d", "MRK_Merck_zscore_60d", "PAYX_Paychex_vol_20d", "TM_Telephone_vol_20d", "EMR_Emerson_ret_20d", "TGT_Target_zscore_60d", "HangSeng_HK_ret_5d", "DAX_Germany_vol_20d", "AMZN_ret_5d", "Michigan_Sentiment_ret_20d"], "is_new": true}, {"model_id": "new_h1_NORMAL_LightGBM_N15_t4", "algo": "LightGBM", "regime": "NORMAL", "horizon": 1, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EFFR_ret_1d", "3M_vol_20d", "TXN_vol_20d", "SLB_Schlumberger_ret_1d", "IYM_BasicMaterials_ret_20d", "heston_var_ev_h7", "HUM_Humana_ret_5d", "QQQ_vol_20d", "HD_zscore_60d", "US3Y_Rate_ret_5d", "vix_acceleration_1d", "DIS_vol_20d", "US3M_Rate_vol_20d"], "is_new": true}, {"model_id": "new_h1_NORMAL_LightGBM_N15_t5", "algo": "LightGBM", "regime": "NORMAL", "horizon": 1, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "Retail_Sales_zscore_60d", "AVB_AvalonBay_zscore_60d", "DHR_ret_1d", "US1Y_Rate_ret_20d", "Core_CPI_zscore_60d", "HangSeng_HK_vol_20d", "AMZN_ret_5d", "M_Macys_vol_20d", "XLY_Disc_vol_20d", "EWL_Switzerland_zscore_60d", "PAYX_Paychex_vol_20d", "LOW_Lowes_ret_20d", "CMCSA_ret_1d"], "is_new": true}, {"model_id": "new_h1_NORMAL_LightGBM_N15_t6", "algo": "LightGBM", "regime": "NORMAL", "horizon": 1, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "HD_ret_1d", "EWC_Canada_zscore_60d", "AMT_AmericanTower_ret_1d", "CI_Cigna_vol_20d", "TM_Telephone_ret_1d", "PAYX_Paychex_vol_20d", "vix_mean_abs_ret_5d", "CLX_Clorox_vol_20d", "SO_SouthernCo_ret_5d", "MS_MorganStanley_ret_5d", "EWA_Australia_zscore_60d", "SBUX_zscore_60d", "INTC_ret_5d"], "is_new": true}, {"model_id": "new_h1_NORMAL_LightGBM_N15_t7", "algo": "LightGBM", "regime": "NORMAL", "horizon": 1, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "DIS_vol_20d", "Nikkei_Japan_vol_20d", "MO_AltriaMG_ret_1d", "MSTR_Bitcoin3_ret_5d", "MRK_Merck_zscore_60d", "AMD_ret_5d", "HD_ret_20d", "LLY_zscore_60d", "LMT_LockheedMartin_ret_1d", "HangSeng_HK_vol_20d", "PAYX_Paychex_vol_20d", "US30Y_Rate_ret_20d", "CLX_Clorox_vol_20d"], "is_new": true}, {"model_id": "new_h1_NORMAL_LightGBM_N20_t0", "algo": "LightGBM", "regime": "NORMAL", "horizon": 1, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "HD_ret_5d", "EMR_Emerson_ret_20d", "NWL_Newell_ret_20d", "NOC_Northrop_ret_20d", "DHR_ret_1d", "VOD_Vodafone_zscore_60d", "WTI_Oil_FRED_zscore_60d", "CLX_Clorox_vol_20d", "DHR_vol_20d", "EWH_HongKong_ret_5d", "HangSeng_HK_ret_5d", "QQQ_vol_20d", "TED_Spread_vol_20d", "IYR_US_REIT2_zscore_60d", "BTI_BritishAmerican_ret_20d", "VRP_ma5", "MSTR_Bitcoin3_ret_1d", "3M_vol_20d"], "is_new": true}, {"model_id": "new_h1_NORMAL_LightGBM_N20_t1", "algo": "LightGBM", "regime": "NORMAL", "horizon": 1, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "US1Y_Rate_ret_5d", "PG_ret_20d", "US3M_Rate_zscore_60d", "DHR_ret_1d", "EWG_Germany_ret_20d", "ES_Evergy_ret_1d", "MSTR_Bitcoin3_ret_20d", "HD_zscore_60d", "US1Y_Rate_ret_20d", "SBUX_zscore_60d", "AVB_AvalonBay_zscore_60d", "HangSeng_HK_ret_5d", "US7Y_Rate_ret_20d", "SCHW_Schwab_ret_5d", "US30Y_Rate_ret_20d", "EWM_Malaysia_ret_1d", "AMZN_ret_5d", "3M_ret_5d"], "is_new": true}, {"model_id": "new_h1_NORMAL_LightGBM_N20_t2", "algo": "LightGBM", "regime": "NORMAL", "horizon": 1, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "DAX_Germany_vol_20d", "spx_abs_ret_max_5d", "vix_acceleration_1d", "HD_ret_5d", "NOC_Northrop_ret_20d", "EQIX_Equinix_ret_5d", "SBUX_zscore_60d", "heston_var_ev_h7", "EWS_Singapore_ret_5d", "AORD_AUS_zscore_60d", "Core_PCE_zscore_60d", "GILD_Gilead_ret_20d", "SCHW_Schwab_ret_5d", "BTI_BritishAmerican_ret_5d", "WTI_Oil_FRED_zscore_60d", "LOW_Lowes_ret_5d", "US6M_Rate_ret_20d", "SBUX_ret_5d"], "is_new": true}, {"model_id": "new_h1_NORMAL_LightGBM_N20_t3", "algo": "LightGBM", "regime": "NORMAL", "horizon": 1, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "Core_PCE_zscore_60d", "MRK_Merck_zscore_60d", "SO_SouthernCo_ret_5d", "US5Y_Rate_ret_5d", "AMGN_Amgen_ret_1d", "EXC_Exelon_zscore_60d", "EMR_Emerson_ret_20d", "US6M_Rate_ret_20d", "BDX_Becton_Dickinson_ret_20d", "DIS_vol_20d", "INTC_ret_5d", "PG_ret_20d", "DE_Deere_vol_20d", "LOW_Lowes_ret_20d", "BA_ret_1d", "GE_ret_1d", "LOW_Lowes_ret_5d", "NVDA_vol_20d"], "is_new": true}, {"model_id": "new_h1_NORMAL_LightGBM_N20_t4", "algo": "LightGBM", "regime": "NORMAL", "horizon": 1, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "HD_zscore_60d", "TED_Spread_zscore_60d", "heston_var_ev_h3", "Brent_Oil_FRED_ret_5d", "SO_SouthernCo_ret_5d", "EWA_Australia_zscore_60d", "XOM_ret_20d", "EQR_Equity_ret_1d", "CPB_CampbellSoup_ret_20d", "WTI_Oil_FRED_zscore_60d", "ENB_EnbridgeInc_ret_1d", "TXN_vol_20d", "ITT_ITTInc_ret_5d", "INTC_ret_1d", "HangSeng_HK_ret_5d", "GE_ret_1d", "LOW_Lowes_ret_5d", "TGT_Target_zscore_60d"], "is_new": true}, {"model_id": "new_h1_NORMAL_LightGBM_N20_t5", "algo": "LightGBM", "regime": "NORMAL", "horizon": 1, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "DHR_ret_1d", "SPY_zscore_60d", "HD_ret_20d", "MRK_Merck_zscore_60d", "SBUX_vol_20d", "NFCI_ret_5d", "JNJ_ret_1d", "EMR_Emerson_ret_20d", "CPB_CampbellSoup_vol_20d", "EWL_Switzerland_vol_20d", "EFFR_ret_1d", "SJM_JM_Smucker_ret_5d", "EWL_Switzerland_zscore_60d", "EWA_Australia_ret_1d", "US30Y_Rate_ret_20d", "XLV_Health_zscore_60d", "TED_Spread_vol_20d", "TGT_Target_zscore_60d"], "is_new": true}, {"model_id": "new_h1_NORMAL_LightGBM_N20_t6", "algo": "LightGBM", "regime": "NORMAL", "horizon": 1, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "Brent_Oil_FRED_ret_5d", "Core_CPI_zscore_60d", "DE_Deere_ret_5d", "AMGN_Amgen_ret_1d", "Retail_Sales_zscore_60d", "LMT_LockheedMartin_ret_1d", "IBEX_Spain_ret_20d", "LOW_Lowes_ret_5d", "DHR_ret_1d", "PAYX_Paychex_ret_20d", "VVIX_ret_20d", "T10Y2Y_Spread_ret_5d", "JNJ_ret_1d", "EOG_EOGResources_vol_20d", "HangSeng_HK_ret_5d", "AORD_AUS_zscore_60d", "Core_PCE_zscore_60d", "EWL_Switzerland_zscore_60d"], "is_new": true}, {"model_id": "new_h1_NORMAL_LightGBM_N20_t7", "algo": "LightGBM", "regime": "NORMAL", "horizon": 1, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "DAX_Germany_zscore_60d", "EWH_HongKong_ret_5d", "DE_Deere_ret_5d", "CCI_CrownCastle_vol_20d", "NWL_Newell_ret_20d", "CTAS_Cintas_vol_20d", "Brent_Oil_FRED_ret_5d", "AMGN_Amgen_ret_1d", "SBUX_ret_5d", "Nikkei_Japan_zscore_60d", "AVB_AvalonBay_zscore_60d", "EWC_Canada_zscore_60d", "CPB_CampbellSoup_zscore_60d", "ASX_Australia_ret_5d", "spx_momentum_3d", "SO_SouthernCo_ret_5d", "VRP_ma5", "BLK_BlackRock_zscore_60d"], "is_new": true}, {"model_id": "new_h1_NORMAL_LightGBM_N25_t0", "algo": "LightGBM", "regime": "NORMAL", "horizon": 1, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "BTI_BritishAmerican_ret_20d", "HangSeng_HK_ret_5d", "AMD_ret_1d", "CPB_CampbellSoup_ret_20d", "DAX_Germany_vol_20d", "XLB_Materials_zscore_60d", "EWG_Germany_vol_20d", "VOD_Vodafone_zscore_60d", "EWG_Germany_ret_20d", "ES_Evergy_ret_1d", "LMT_LockheedMartin_ret_1d", "AORD_AUS_zscore_60d", "CMCSA_ret_1d", "DIS_vol_20d", "HD_ret_1d", "US6M_Rate_ret_20d", "gjr_condvar_h1", "LMT_LockheedMartin_vol_20d", "EMR_Emerson_ret_20d", "SJM_JM_Smucker_ret_5d", "EWQ_France_zscore_60d", "SLB_Schlumberger_ret_1d", "US3M_Rate_zscore_60d"], "is_new": true}, {"model_id": "new_h1_NORMAL_LightGBM_N25_t1", "algo": "LightGBM", "regime": "NORMAL", "horizon": 1, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWA_Australia_zscore_60d", "HD_ret_20d", "EOG_EOGResources_ret_5d", "TM_Telephone_ret_1d", "3M_vol_20d", "CPB_CampbellSoup_ret_5d", "EWJ_Japan_vol_20d", "SLB_Schlumberger_ret_1d", "QQQ_vol_20d", "LUV_SouthwestAir_ret_5d", "VOD_Vodafone_zscore_60d", "ES_Evergy_ret_1d", "ORCL_zscore_60d", "CPB_CampbellSoup_zscore_60d", "EQR_Equity_ret_1d", "LMT_LockheedMartin_vol_20d", "MSTR_Bitcoin3_ret_20d", "NWL_Newell_ret_20d", "heston_var_ev_h7", "SO_SouthernCo_ret_5d", "EWM_Malaysia_vol_20d", "NEE_NextEra_ret_20d", "CTAS_Cintas_vol_20d"], "is_new": true}, {"model_id": "new_h1_NORMAL_LightGBM_N25_t2", "algo": "LightGBM", "regime": "NORMAL", "horizon": 1, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "T10Y2Y_Spread_ret_5d", "DE_Deere_vol_20d", "SCHW_Schwab_ret_5d", "LOW_Lowes_ret_5d", "HD_ret_5d", "US1Y_Rate_ret_5d", "AVB_AvalonBay_zscore_60d", "CI_Cigna_vol_20d", "IWM_SmallCap_vol_20d", "spx_vol_5d", "Core_CPI_zscore_60d", "MSTR_Bitcoin3_ret_5d", "HD_ret_20d", "PCAR_PaccarInc_ret_5d", "WTI_Oil_FRED_zscore_60d", "IBEX_Spain_ret_20d", "PAYX_Paychex_ret_20d", "EWM_Malaysia_ret_1d", "US1Y_Rate_ret_20d", "NEE_NextEra_ret_20d", "HD_zscore_60d", "Nikkei_Japan_vol_20d", "EWA_Australia_ret_1d"], "is_new": true}, {"model_id": "new_h1_NORMAL_LightGBM_N25_t3", "algo": "LightGBM", "regime": "NORMAL", "horizon": 1, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "AMD_ret_5d", "SBUX_zscore_60d", "CPB_CampbellSoup_vol_20d", "MS_MorganStanley_zscore_60d", "ORCL_zscore_60d", "T10Y2Y_Spread_ret_5d", "CI_Cigna_vol_20d", "Core_PCE_zscore_60d", "DHR_vol_20d", "spx_momentum_3d", "M_Macys_vol_20d", "VRP_ma5", "spx_vol_5d", "EOG_EOGResources_ret_5d", "BTI_BritishAmerican_ret_20d", "LUV_SouthwestAir_ret_5d", "EWC_Canada_zscore_60d", "Industrial_Production_zscore_60d", "EWG_Germany_ret_20d", "VVIX_ret_20d", "Michigan_Sentiment_ret_20d", "DAX_Germany_vol_20d", "EWM_Malaysia_zscore_60d"], "is_new": true}, {"model_id": "new_h1_NORMAL_LightGBM_N25_t4", "algo": "LightGBM", "regime": "NORMAL", "horizon": 1, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "heston_var_ev_h5", "M_Macys_vol_20d", "LOW_Lowes_ret_20d", "QQQ_vol_20d", "XLB_Materials_zscore_60d", "Michigan_Sentiment_ret_20d", "SBUX_vol_20d", "SLB_Schlumberger_ret_5d", "HD_ret_1d", "DIS_vol_20d", "PCAR_PaccarInc_ret_5d", "CPB_CampbellSoup_zscore_60d", "DAX_Germany_vol_20d", "EWY_Korea_zscore_60d", "SBUX_zscore_60d", "SJM_JM_Smucker_ret_1d", "EWM_Malaysia_zscore_60d", "DE_Deere_vol_20d", "AORD_AUS_zscore_60d", "CPB_CampbellSoup_ret_5d", "BA_ret_1d", "BTI_BritishAmerican_ret_5d", "XLK_Tech_zscore_60d"], "is_new": true}, {"model_id": "new_h1_NORMAL_LightGBM_N25_t5", "algo": "LightGBM", "regime": "NORMAL", "horizon": 1, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "XLB_Materials_zscore_60d", "IWM_SmallCap_vol_20d", "EFFR_ret_1d", "HangSeng_HK_ret_5d", "XOM_ret_1d", "spx_abs_ret_max_5d", "CPB_CampbellSoup_ret_20d", "Brent_Oil_FRED_ret_5d", "T10Y2Y_Spread_ret_5d", "SCHW_Schwab_ret_5d", "PLD_Prologis_ret_5d", "EWG_Germany_ret_20d", "HUM_Humana_ret_5d", "SJM_JM_Smucker_ret_1d", "SBUX_zscore_60d", "SLB_Schlumberger_ret_1d", "EXC_Exelon_zscore_60d", "TM_Telephone_vol_20d", "JNJ_ret_1d", "vix_mean_abs_ret_5d", "TXN_vol_20d", "AMGN_Amgen_ret_1d", "ASX_Australia_ret_5d"], "is_new": true}, {"model_id": "new_h1_NORMAL_LightGBM_N25_t6", "algo": "LightGBM", "regime": "NORMAL", "horizon": 1, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "INTC_ret_1d", "MSTR_Bitcoin3_ret_1d", "PAYX_Paychex_ret_20d", "PFE_ret_1d", "ASX_Australia_vol_20d", "ASX_Australia_ret_5d", "EQIX_Equinix_ret_5d", "US1Y_Rate_ret_5d", "HangSeng_HK_ret_5d", "ENB_EnbridgeInc_ret_1d", "SBUX_zscore_60d", "Nikkei_Japan_zscore_60d", "DE_Deere_vol_20d", "IBEX_Spain_ret_20d", "MS_MorganStanley_ret_1d", "SBUX_vol_20d", "CTAS_Cintas_vol_20d", "US3M_Rate_zscore_60d", "gjr_condvar_h1", "US1Y_Rate_ret_20d", "TM_Telephone_vol_20d", "US3Y_Rate_ret_5d", "PPL_PPL_ret_1d"], "is_new": true}, {"model_id": "new_h1_NORMAL_LightGBM_N25_t7", "algo": "LightGBM", "regime": "NORMAL", "horizon": 1, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "FedFunds_zscore_60d", "SCHW_Schwab_ret_5d", "VRP_ma5", "MSTR_Bitcoin3_ret_1d", "XLV_Health_zscore_60d", "LMT_LockheedMartin_vol_20d", "HD_ret_20d", "ITT_ITTInc_ret_5d", "BLK_BlackRock_zscore_60d", "AORD_AUS_zscore_60d", "spx_vol_5d", "AXP_Amex_ret_20d", "DAX_Germany_zscore_60d", "LUV_SouthwestAir_ret_5d", "LMT_LockheedMartin_ret_1d", "US7Y_Rate_ret_20d", "AMZN_ret_5d", "JNJ_ret_1d", "MSTR_Bitcoin3_ret_20d", "CPB_CampbellSoup_ret_20d", "Nikkei_Japan_vol_20d", "LLY_zscore_60d", "LOW_Lowes_ret_20d"], "is_new": true}, {"model_id": "new_h1_NORMAL_LightGBM_N30_t0", "algo": "LightGBM", "regime": "NORMAL", "horizon": 1, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWY_Korea_ret_20d", "EWQ_France_ret_20d", "heston_var_ev_h7", "IBEX_Spain_ret_20d", "VRP_ma5", "CI_Cigna_vol_20d", "BTI_BritishAmerican_ret_20d", "HangSeng_HK_vol_20d", "Core_CPI_zscore_60d", "Industrial_Production_zscore_60d", "EOG_EOGResources_ret_5d", "AMT_AmericanTower_ret_1d", "TM_Telephone_ret_1d", "IWM_SmallCap_vol_20d", "ASX_Australia_vol_20d", "PFE_ret_1d", "NWL_Newell_ret_20d", "US3M_Rate_zscore_60d", "SO_SouthernCo_ret_5d", "NVDA_vol_20d", "GD_GeneralDynamics_zscore_60d", "XOM_ret_20d", "PPL_PPL_ret_1d", "VOD_Vodafone_zscore_60d", "INTC_ret_5d", "MSTR_Bitcoin3_ret_1d", "MS_MorganStanley_ret_1d", "ITT_ITTInc_ret_5d"], "is_new": true}, {"model_id": "new_h1_NORMAL_LightGBM_N30_t1", "algo": "LightGBM", "regime": "NORMAL", "horizon": 1, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWG_Germany_vol_20d", "Industrial_Production_zscore_60d", "SJM_JM_Smucker_ret_1d", "SJM_JM_Smucker_ret_5d", "vix_acceleration_1d", "ENB_EnbridgeInc_ret_1d", "XLV_Health_zscore_60d", "HangSeng_HK_ret_1d", "LUV_SouthwestAir_ret_5d", "ORCL_vol_20d", "EWG_Germany_ret_20d", "spx_vol_5d", "Brent_Oil_FRED_ret_5d", "TM_Telephone_ret_1d", "EWJ_Japan_vol_20d", "HD_ret_5d", "FedFunds_zscore_60d", "AXP_Amex_vol_20d", "heston_var_ev_h5", "US3Y_Rate_ret_5d", "M_Macys_vol_20d", "EWM_Malaysia_ret_1d", "CMCSA_ret_1d", "US6M_Rate_ret_20d", "CPB_CampbellSoup_ret_20d", "spx_abs_ret_max_5d", "GE_ret_1d", "BTI_BritishAmerican_ret_5d"], "is_new": true}, {"model_id": "new_h1_NORMAL_LightGBM_N30_t2", "algo": "LightGBM", "regime": "NORMAL", "horizon": 1, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "LLY_zscore_60d", "Brent_Oil_FRED_ret_5d", "hmm_p_stress", "HangSeng_HK_ret_5d", "SLB_Schlumberger_ret_1d", "HangSeng_HK_ret_1d", "Nikkei_Japan_zscore_60d", "TM_Telephone_vol_20d", "INTC_ret_1d", "EWG_Germany_ret_20d", "NOC_Northrop_ret_20d", "M_Macys_vol_20d", "EWY_Korea_zscore_60d", "SO_SouthernCo_ret_5d", "PCAR_PaccarInc_ret_5d", "T_ret_1d", "EQR_Equity_ret_1d", "AMGN_Amgen_ret_1d", "SPY_zscore_60d", "ES_Evergy_ret_1d", "US5Y_Rate_ret_5d", "SJM_JM_Smucker_ret_1d", "EWJ_Japan_vol_20d", "PLD_Prologis_ret_5d", "GD_GeneralDynamics_zscore_60d", "TGT_Target_zscore_60d", "ORCL_vol_20d", "US3M_Rate_vol_20d"], "is_new": true}, {"model_id": "new_h1_NORMAL_LightGBM_N30_t3", "algo": "LightGBM", "regime": "NORMAL", "horizon": 1, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "PG_ret_20d", "DIS_vol_20d", "EWQ_France_zscore_60d", "DOW_Price_zscore_60d", "HangSeng_HK_ret_1d", "ORCL_vol_20d", "NFCI_ret_5d", "INTC_ret_5d", "BTI_BritishAmerican_ret_20d", "heston_ev_h3", "EWG_Germany_vol_20d", "heston_var_ev_h7", "GD_GeneralDynamics_zscore_60d", "HD_ret_5d", "EMR_Emerson_ret_20d", "EXC_Exelon_ret_1d", "WTI_Oil_FRED_zscore_60d", "EWQ_France_ret_20d", "Michigan_Sentiment_ret_20d", "spx_vol_5d", "Core_CPI_zscore_60d", "spx_momentum_3d", "EWA_Australia_zscore_60d", "AORD_AUS_zscore_60d", "EWS_Singapore_ret_5d", "IBEX_Spain_ret_20d", "vix_mean_abs_ret_5d", "SBUX_vol_20d"], "is_new": true}, {"model_id": "new_h1_NORMAL_LightGBM_N30_t4", "algo": "LightGBM", "regime": "NORMAL", "horizon": 1, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWG_Germany_ret_20d", "EFFR_vol_20d", "GE_ret_1d", "DE_Deere_vol_20d", "AORD_AUS_zscore_60d", "EXC_Exelon_zscore_60d", "HD_ret_1d", "T_ret_1d", "EQIX_Equinix_ret_5d", "AVB_AvalonBay_zscore_60d", "SBUX_vol_20d", "AMT_AmericanTower_ret_1d", "BDX_Becton_Dickinson_ret_20d", "CI_Cigna_vol_20d", "INTC_ret_1d", "XLB_Materials_zscore_60d", "HangSeng_HK_ret_1d", "BTI_BritishAmerican_ret_20d", "NEE_NextEra_ret_20d", "LUV_SouthwestAir_ret_5d", "CMCSA_ret_1d", "MS_MorganStanley_zscore_60d", "DHR_ret_1d", "EWQ_France_ret_20d", "CTAS_Cintas_vol_20d", "US3Y_Rate_ret_5d", "SJM_JM_Smucker_ret_5d", "Core_PCE_zscore_60d"], "is_new": true}, {"model_id": "new_h1_NORMAL_LightGBM_N30_t5", "algo": "LightGBM", "regime": "NORMAL", "horizon": 1, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWQ_France_ret_20d", "WTI_Oil_FRED_zscore_60d", "DAX_Germany_vol_20d", "INTC_ret_1d", "IYM_BasicMaterials_ret_20d", "US7Y_Rate_ret_20d", "MS_MorganStanley_ret_1d", "vix_mean_abs_ret_5d", "NWL_Newell_ret_20d", "EWL_Switzerland_vol_20d", "DHR_vol_20d", "CPB_CampbellSoup_zscore_60d", "EOG_EOGResources_vol_20d", "EWJ_Japan_vol_20d", "GILD_Gilead_ret_20d", "ORCL_zscore_60d", "NEE_NextEra_ret_20d", "EXC_Exelon_ret_1d", "VOD_Vodafone_zscore_60d", "BDX_Becton_Dickinson_ret_20d", "SBUX_zscore_60d", "TGT_Target_zscore_60d", "MRK_Merck_zscore_60d", "CTAS_Cintas_vol_20d", "INTC_ret_5d", "TXN_vol_20d", "heston_var_ev_h3", "QQQ_vol_20d"], "is_new": true}, {"model_id": "new_h1_NORMAL_LightGBM_N30_t6", "algo": "LightGBM", "regime": "NORMAL", "horizon": 1, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "LOW_Lowes_ret_5d", "QQQ_vol_20d", "MO_AltriaMG_ret_1d", "HangSeng_HK_vol_20d", "AMT_AmericanTower_ret_1d", "heston_var_ev_h7", "XLF_Fin_vol_20d", "ASX_Australia_ret_5d", "BA_ret_1d", "EWQ_France_zscore_60d", "EWY_Korea_zscore_60d", "US1Y_Rate_ret_20d", "MSTR_Bitcoin3_ret_5d", "IBEX_Spain_ret_20d", "vix_mean_abs_ret_5d", "DAX_Germany_vol_20d", "TGT_Target_zscore_60d", "EWQ_France_ret_20d", "PAYX_Paychex_ret_20d", "MS_MorganStanley_ret_1d", "AMD_ret_1d", "MS_MorganStanley_ret_5d", "EXC_Exelon_ret_1d", "HangSeng_HK_ret_5d", "CMCSA_ret_1d", "XLB_Materials_zscore_60d", "HangSeng_HK_ret_1d", "EWH_HongKong_ret_5d"], "is_new": true}, {"model_id": "new_h1_NORMAL_LightGBM_N30_t7", "algo": "LightGBM", "regime": "NORMAL", "horizon": 1, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "ASX_Australia_ret_5d", "GILD_Gilead_ret_20d", "CCI_CrownCastle_vol_20d", "SBUX_zscore_60d", "PPL_PPL_ret_1d", "NFCI_ret_5d", "IYR_US_REIT2_zscore_60d", "NWL_Newell_ret_20d", "CPB_CampbellSoup_ret_5d", "EMR_Emerson_ret_20d", "EXC_Exelon_zscore_60d", "SLB_Schlumberger_ret_1d", "BTI_BritishAmerican_ret_20d", "LMT_LockheedMartin_vol_20d", "NVDA_vol_20d", "XLB_Materials_zscore_60d", "T_ret_1d", "IWM_SmallCap_vol_20d", "PCAR_PaccarInc_ret_5d", "AORD_AUS_zscore_60d", "BA_ret_1d", "ES_Evergy_ret_1d", "PAYX_Paychex_ret_20d", "DE_Deere_vol_20d", "3M_ret_5d", "M_Macys_vol_20d", "EWG_Germany_ret_20d", "HangSeng_HK_vol_20d"], "is_new": true}, {"model_id": "new_h1_NORMAL_GradientBoosting_N5_t0", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 1, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWY_Korea_ret_20d", "SPY_zscore_60d", "hmm_p_stress"], "is_new": true}, {"model_id": "new_h1_NORMAL_GradientBoosting_N5_t1", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 1, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "Retail_Sales_zscore_60d", "DE_Deere_vol_20d", "EXC_Exelon_zscore_60d"], "is_new": true}, {"model_id": "new_h1_NORMAL_GradientBoosting_N5_t2", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 1, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "CI_Cigna_vol_20d", "NOC_Northrop_ret_20d", "MO_AltriaMG_ret_1d"], "is_new": true}, {"model_id": "new_h1_NORMAL_GradientBoosting_N5_t3", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 1, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "AMT_AmericanTower_ret_1d", "US1Y_Rate_ret_5d", "heston_var_ev_h7"], "is_new": true}, {"model_id": "new_h1_NORMAL_GradientBoosting_N5_t4", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 1, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWC_Canada_zscore_60d", "FedFunds_zscore_60d", "EOG_EOGResources_ret_5d"], "is_new": true}, {"model_id": "new_h1_NORMAL_GradientBoosting_N5_t5", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 1, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "DOW_Price_zscore_60d", "CLX_Clorox_vol_20d", "MO_AltriaMG_ret_1d"], "is_new": true}, {"model_id": "new_h1_NORMAL_GradientBoosting_N5_t6", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 1, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "PCAR_PaccarInc_ret_5d", "SCHW_Schwab_ret_5d", "Core_PCE_zscore_60d"], "is_new": true}, {"model_id": "new_h1_NORMAL_GradientBoosting_N5_t7", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 1, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "HD_ret_5d", "SCHW_Schwab_ret_5d", "MSTR_Bitcoin3_ret_20d"], "is_new": true}, {"model_id": "new_h1_NORMAL_GradientBoosting_N8_t0", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 1, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "HD_ret_20d", "EWA_Australia_ret_1d", "NVDA_vol_20d", "EQR_Equity_ret_1d", "NEE_NextEra_ret_20d", "DIS_vol_20d"], "is_new": true}, {"model_id": "new_h1_NORMAL_GradientBoosting_N8_t1", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 1, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "XLY_Disc_vol_20d", "PAYX_Paychex_zscore_60d", "EWY_Korea_ret_20d", "MO_AltriaMG_ret_1d", "US7Y_Rate_ret_20d", "heston_var_ev_h7"], "is_new": true}, {"model_id": "new_h1_NORMAL_GradientBoosting_N8_t2", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 1, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "DE_Deere_vol_20d", "ITT_ITTInc_ret_5d", "MS_MorganStanley_zscore_60d", "EWJ_Japan_vol_20d", "MSTR_Bitcoin3_ret_5d", "NWL_Newell_ret_20d"], "is_new": true}, {"model_id": "new_h1_NORMAL_GradientBoosting_N8_t3", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 1, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "US1Y_Rate_ret_5d", "BLK_BlackRock_zscore_60d", "BA_ret_1d", "IYR_US_REIT2_zscore_60d", "EWQ_France_ret_20d", "PPL_PPL_ret_1d"], "is_new": true}, {"model_id": "new_h1_NORMAL_GradientBoosting_N8_t4", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 1, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "SLB_Schlumberger_ret_1d", "XLY_Disc_vol_20d", "MSTR_Bitcoin3_ret_20d", "HangSeng_HK_ret_5d", "MS_MorganStanley_zscore_60d", "vix_mean_abs_ret_5d"], "is_new": true}, {"model_id": "new_h1_NORMAL_GradientBoosting_N8_t5", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 1, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EXC_Exelon_zscore_60d", "EOG_EOGResources_ret_5d", "SJM_JM_Smucker_ret_5d", "SCHW_Schwab_ret_5d", "Industrial_Production_zscore_60d", "PLD_Prologis_ret_5d"], "is_new": true}, {"model_id": "new_h1_NORMAL_GradientBoosting_N8_t6", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 1, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "DHR_vol_20d", "EXC_Exelon_zscore_60d", "Core_CPI_zscore_60d", "INTC_ret_1d", "AXP_Amex_ret_20d", "DAX_Germany_zscore_60d"], "is_new": true}, {"model_id": "new_h1_NORMAL_GradientBoosting_N8_t7", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 1, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "HD_ret_20d", "FedFunds_zscore_60d", "XLK_Tech_zscore_60d", "SPY_zscore_60d", "INTC_ret_1d", "ORCL_vol_20d"], "is_new": true}, {"model_id": "new_h1_NORMAL_GradientBoosting_N10_t0", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 1, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "US1Y_Rate_ret_5d", "spx_momentum_3d", "heston_ev_h3", "ORCL_zscore_60d", "BA_ret_1d", "EWA_Australia_ret_1d", "DOW_Price_zscore_60d", "EWM_Malaysia_vol_20d"], "is_new": true}, {"model_id": "new_h1_NORMAL_GradientBoosting_N10_t1", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 1, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EMR_Emerson_ret_20d", "PLD_Prologis_ret_5d", "ORCL_vol_20d", "Core_CPI_zscore_60d", "SCHW_Schwab_ret_5d", "AVB_AvalonBay_zscore_60d", "PAYX_Paychex_ret_20d", "LOW_Lowes_ret_20d"], "is_new": true}, {"model_id": "new_h1_NORMAL_GradientBoosting_N10_t2", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 1, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "CTAS_Cintas_vol_20d", "DIS_vol_20d", "DAX_Germany_vol_20d", "LMT_LockheedMartin_ret_1d", "CMCSA_ret_1d", "EWG_Germany_vol_20d", "HD_ret_20d", "EQR_Equity_ret_1d"], "is_new": true}, {"model_id": "new_h1_NORMAL_GradientBoosting_N10_t3", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 1, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "TED_Spread_vol_20d", "US3Y_Rate_ret_5d", "EQR_Equity_ret_1d", "NOC_Northrop_ret_20d", "Brent_Oil_FRED_ret_5d", "EWY_Korea_ret_20d", "SJM_JM_Smucker_ret_1d", "CPB_CampbellSoup_ret_20d"], "is_new": true}, {"model_id": "new_h1_NORMAL_GradientBoosting_N10_t4", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 1, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "MS_MorganStanley_ret_5d", "MRK_Merck_zscore_60d", "VRP_ma5", "CPB_CampbellSoup_zscore_60d", "ITT_ITTInc_ret_5d", "EFFR_ret_1d", "AORD_AUS_zscore_60d", "CCI_CrownCastle_vol_20d"], "is_new": true}, {"model_id": "new_h1_NORMAL_GradientBoosting_N10_t5", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 1, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "VVIX_ret_20d", "spx_momentum_3d", "3M_vol_20d", "IYM_BasicMaterials_ret_20d", "AXP_Amex_ret_20d", "spx_abs_ret_max_5d", "XOM_ret_1d", "PAYX_Paychex_zscore_60d"], "is_new": true}, {"model_id": "new_h1_NORMAL_GradientBoosting_N10_t6", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 1, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "heston_var_ev_h5", "HD_ret_20d", "ITT_ITTInc_ret_5d", "BDX_Becton_Dickinson_ret_20d", "Brent_Oil_FRED_ret_20d", "T_ret_1d", "XLB_Materials_zscore_60d", "MSTR_Bitcoin3_ret_20d"], "is_new": true}, {"model_id": "new_h1_NORMAL_GradientBoosting_N10_t7", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 1, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWQ_France_zscore_60d", "M_Macys_vol_20d", "CLX_Clorox_vol_20d", "AMT_AmericanTower_ret_1d", "EWH_HongKong_ret_5d", "EMR_Emerson_ret_20d", "MS_MorganStanley_ret_1d", "Michigan_Sentiment_ret_20d"], "is_new": true}, {"model_id": "new_h1_NORMAL_GradientBoosting_N12_t0", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 1, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "SBUX_zscore_60d", "Nikkei_Japan_zscore_60d", "IBEX_Spain_ret_20d", "XLB_Materials_zscore_60d", "DE_Deere_ret_5d", "LOW_Lowes_ret_20d", "vix_acceleration_1d", "CPB_CampbellSoup_ret_20d", "MO_AltriaMG_ret_1d", "SO_SouthernCo_ret_5d"], "is_new": true}, {"model_id": "new_h1_NORMAL_GradientBoosting_N12_t1", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 1, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "QQQ_vol_20d", "GD_GeneralDynamics_zscore_60d", "US3M_Rate_vol_20d", "WTI_Oil_FRED_zscore_60d", "NWL_Newell_ret_20d", "CMCSA_ret_1d", "MO_AltriaMG_ret_1d", "XLV_Health_zscore_60d", "ES_Evergy_ret_1d", "ORCL_vol_20d"], "is_new": true}, {"model_id": "new_h1_NORMAL_GradientBoosting_N12_t2", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 1, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "SO_SouthernCo_ret_5d", "JNJ_ret_1d", "PG_ret_20d", "EWM_Malaysia_ret_1d", "heston_var_ev_h7", "EFFR_ret_1d", "SCHW_Schwab_ret_5d", "TED_Spread_vol_20d", "gjr_condvar_h1", "CI_Cigna_vol_20d"], "is_new": true}, {"model_id": "new_h1_NORMAL_GradientBoosting_N12_t3", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 1, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "Retail_Sales_zscore_60d", "HangSeng_HK_ret_1d", "DOW_Price_zscore_60d", "DAX_Germany_zscore_60d", "EWY_Korea_ret_20d", "T10Y2Y_Spread_ret_5d", "Nikkei_Japan_vol_20d", "HUM_Humana_ret_5d", "US3M_Rate_zscore_60d", "AXP_Amex_vol_20d"], "is_new": true}, {"model_id": "new_h1_NORMAL_GradientBoosting_N12_t4", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 1, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "SLB_Schlumberger_ret_5d", "NWL_Newell_ret_20d", "NFCI_ret_5d", "CPB_CampbellSoup_ret_20d", "PG_ret_20d", "spx_vol_5d", "US3M_Rate_zscore_60d", "EWY_Korea_zscore_60d", "LMT_LockheedMartin_vol_20d", "EXC_Exelon_ret_1d"], "is_new": true}, {"model_id": "new_h1_NORMAL_GradientBoosting_N12_t5", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 1, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "spx_vol_5d", "PG_ret_20d", "EQIX_Equinix_ret_5d", "EQR_Equity_ret_1d", "MS_MorganStanley_ret_5d", "AXP_Amex_ret_20d", "CI_Cigna_vol_20d", "NOC_Northrop_ret_20d", "HangSeng_HK_ret_1d", "INTC_ret_1d"], "is_new": true}, {"model_id": "new_h1_NORMAL_GradientBoosting_N12_t6", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 1, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "JNJ_ret_1d", "ES_Evergy_ret_1d", "EWJ_Japan_vol_20d", "CCI_CrownCastle_vol_20d", "XLB_Materials_zscore_60d", "EXC_Exelon_zscore_60d", "CPB_CampbellSoup_zscore_60d", "ORCL_vol_20d", "TGT_Target_zscore_60d", "AMGN_Amgen_ret_1d"], "is_new": true}, {"model_id": "new_h1_NORMAL_GradientBoosting_N12_t7", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 1, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EOG_EOGResources_ret_5d", "spx_abs_ret_max_5d", "Retail_Sales_zscore_60d", "PFE_ret_1d", "Michigan_Sentiment_ret_20d", "spx_momentum_3d", "XLB_Materials_zscore_60d", "SBUX_vol_20d", "HangSeng_HK_ret_5d", "XLY_Disc_vol_20d"], "is_new": true}, {"model_id": "new_h1_NORMAL_GradientBoosting_N15_t0", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 1, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "CLX_Clorox_vol_20d", "DAX_Germany_zscore_60d", "DE_Deere_vol_20d", "AMD_ret_5d", "EWA_Australia_ret_1d", "CTAS_Cintas_vol_20d", "AMGN_Amgen_ret_1d", "AMZN_ret_5d", "EWH_HongKong_ret_5d", "SBUX_ret_5d", "hmm_p_stress", "DIS_vol_20d", "EWQ_France_zscore_60d"], "is_new": true}, {"model_id": "new_h1_NORMAL_GradientBoosting_N15_t1", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 1, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "PG_ret_20d", "DIS_vol_20d", "QQQ_vol_20d", "NWL_Newell_ret_20d", "AMT_AmericanTower_ret_1d", "Core_CPI_zscore_60d", "ENB_EnbridgeInc_ret_1d", "EFFR_ret_1d", "PAYX_Paychex_ret_20d", "SBUX_vol_20d", "PPL_PPL_ret_1d", "PAYX_Paychex_zscore_60d", "US7Y_Rate_ret_20d"], "is_new": true}, {"model_id": "new_h1_NORMAL_GradientBoosting_N15_t2", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 1, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "BA_ret_1d", "IYR_US_REIT2_zscore_60d", "AMD_ret_1d", "CPB_CampbellSoup_vol_20d", "gjr_condvar_h1", "Nikkei_Japan_zscore_60d", "NOC_Northrop_ret_20d", "SCHW_Schwab_ret_5d", "US6M_Rate_ret_20d", "VVIX_ret_20d", "US1Y_Rate_ret_20d", "HD_zscore_60d", "ASX_Australia_vol_20d"], "is_new": true}, {"model_id": "new_h1_NORMAL_GradientBoosting_N15_t3", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 1, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWM_Malaysia_zscore_60d", "BLK_BlackRock_zscore_60d", "TED_Spread_zscore_60d", "AMGN_Amgen_ret_1d", "NWL_Newell_ret_20d", "US5Y_Rate_ret_5d", "INTC_ret_1d", "US1Y_Rate_ret_20d", "BTI_BritishAmerican_ret_20d", "GD_GeneralDynamics_zscore_60d", "heston_var_ev_h7", "NOC_Northrop_ret_20d", "QQQ_vol_20d"], "is_new": true}, {"model_id": "new_h1_NORMAL_GradientBoosting_N15_t4", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 1, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "ORCL_vol_20d", "AMT_AmericanTower_ret_1d", "EMR_Emerson_ret_20d", "HD_ret_5d", "EWA_Australia_zscore_60d", "PPL_PPL_ret_1d", "LLY_zscore_60d", "ORCL_zscore_60d", "EWM_Malaysia_zscore_60d", "SLB_Schlumberger_ret_1d", "EWL_Switzerland_zscore_60d", "VOD_Vodafone_zscore_60d", "Industrial_Production_zscore_60d"], "is_new": true}, {"model_id": "new_h1_NORMAL_GradientBoosting_N15_t5", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 1, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "MS_MorganStanley_ret_1d", "NOC_Northrop_ret_20d", "spx_abs_ret_max_5d", "LOW_Lowes_ret_5d", "PPL_PPL_ret_1d", "EMR_Emerson_ret_20d", "XOM_ret_1d", "EWH_HongKong_ret_5d", "HD_ret_20d", "CLX_Clorox_vol_20d", "vix_acceleration_1d", "SBUX_ret_5d", "DE_Deere_ret_5d"], "is_new": true}, {"model_id": "new_h1_NORMAL_GradientBoosting_N15_t6", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 1, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "CPB_CampbellSoup_ret_5d", "NOC_Northrop_ret_20d", "SCHW_Schwab_ret_5d", "HangSeng_HK_ret_5d", "EWQ_France_ret_20d", "EWH_HongKong_ret_5d", "US1Y_Rate_ret_5d", "AXP_Amex_vol_20d", "AMGN_Amgen_ret_1d", "LOW_Lowes_ret_20d", "EXC_Exelon_zscore_60d", "EFFR_vol_20d", "EOG_EOGResources_vol_20d"], "is_new": true}, {"model_id": "new_h1_NORMAL_GradientBoosting_N15_t7", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 1, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "Nikkei_Japan_zscore_60d", "Core_CPI_zscore_60d", "XLY_Disc_vol_20d", "EWM_Malaysia_vol_20d", "Core_PCE_zscore_60d", "US30Y_Rate_ret_20d", "EWM_Malaysia_ret_1d", "SBUX_vol_20d", "EXC_Exelon_zscore_60d", "JNJ_ret_1d", "LMT_LockheedMartin_vol_20d", "AXP_Amex_ret_20d", "US1Y_Rate_ret_5d"], "is_new": true}, {"model_id": "new_h1_NORMAL_GradientBoosting_N20_t0", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 1, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "ORCL_zscore_60d", "PFE_ret_1d", "CTAS_Cintas_vol_20d", "CPB_CampbellSoup_zscore_60d", "XLK_Tech_zscore_60d", "CPB_CampbellSoup_vol_20d", "US30Y_Rate_ret_20d", "AVB_AvalonBay_zscore_60d", "SLB_Schlumberger_ret_1d", "ENB_EnbridgeInc_ret_1d", "gjr_condvar_h1", "Core_PCE_zscore_60d", "EWG_Germany_vol_20d", "LOW_Lowes_ret_5d", "EFFR_vol_20d", "HD_zscore_60d", "PAYX_Paychex_ret_20d", "SJM_JM_Smucker_ret_1d"], "is_new": true}, {"model_id": "new_h1_NORMAL_GradientBoosting_N20_t1", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 1, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "IYR_US_REIT2_zscore_60d", "DHR_ret_1d", "DAX_Germany_vol_20d", "EWG_Germany_ret_20d", "ORCL_zscore_60d", "NOC_Northrop_ret_20d", "3M_vol_20d", "EWS_Singapore_ret_5d", "QQQ_vol_20d", "JNJ_ret_1d", "HD_ret_5d", "SLB_Schlumberger_ret_1d", "EOG_EOGResources_ret_5d", "CI_Cigna_vol_20d", "GD_GeneralDynamics_zscore_60d", "MRK_Merck_zscore_60d", "LOW_Lowes_ret_5d", "gjr_condvar_h1"], "is_new": true}, {"model_id": "new_h1_NORMAL_GradientBoosting_N20_t2", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 1, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWM_Malaysia_vol_20d", "DHR_ret_1d", "Nikkei_Japan_vol_20d", "NFCI_ret_5d", "NEE_NextEra_ret_20d", "PFE_ret_1d", "SPY_zscore_60d", "LOW_Lowes_ret_5d", "AORD_AUS_zscore_60d", "PAYX_Paychex_ret_20d", "DIS_vol_20d", "AMD_ret_1d", "BTI_BritishAmerican_ret_5d", "CI_Cigna_vol_20d", "Industrial_Production_zscore_60d", "INTC_ret_1d", "M_Macys_vol_20d", "ENB_EnbridgeInc_ret_1d"], "is_new": true}, {"model_id": "new_h1_NORMAL_GradientBoosting_N20_t3", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 1, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "spx_vol_5d", "heston_var_ev_h5", "DOW_Price_zscore_60d", "US5Y_Rate_ret_5d", "BDX_Becton_Dickinson_ret_20d", "gjr_condvar_h1", "T10Y2Y_Spread_ret_5d", "HUM_Humana_ret_5d", "NVDA_vol_20d", "M_Macys_vol_20d", "US1Y_Rate_ret_20d", "HangSeng_HK_ret_5d", "INTC_ret_1d", "LLY_zscore_60d", "AMD_ret_1d", "HD_ret_1d", "TED_Spread_zscore_60d", "XLK_Tech_zscore_60d"], "is_new": true}, {"model_id": "new_h1_NORMAL_GradientBoosting_N20_t4", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 1, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "TXN_vol_20d", "XOM_ret_1d", "XLY_Disc_vol_20d", "hmm_p_stress", "PAYX_Paychex_ret_20d", "LLY_zscore_60d", "Michigan_Sentiment_ret_20d", "EWC_Canada_zscore_60d", "HangSeng_HK_ret_5d", "CI_Cigna_vol_20d", "ASX_Australia_vol_20d", "TM_Telephone_ret_1d", "IYR_US_REIT2_zscore_60d", "MS_MorganStanley_ret_1d", "DAX_Germany_zscore_60d", "T10Y2Y_Spread_ret_5d", "TM_Telephone_vol_20d", "HD_ret_20d"], "is_new": true}, {"model_id": "new_h1_NORMAL_GradientBoosting_N20_t5", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 1, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "heston_ev_h3", "EFFR_vol_20d", "AMT_AmericanTower_ret_1d", "AORD_AUS_zscore_60d", "TM_Telephone_vol_20d", "VRP_ma5", "EXC_Exelon_zscore_60d", "EWQ_France_zscore_60d", "HangSeng_HK_ret_5d", "NWL_Newell_ret_20d", "SLB_Schlumberger_ret_1d", "CTAS_Cintas_vol_20d", "BLK_BlackRock_zscore_60d", "MRK_Merck_zscore_60d", "XOM_ret_20d", "LMT_LockheedMartin_vol_20d", "DAX_Germany_vol_20d", "AXP_Amex_vol_20d"], "is_new": true}, {"model_id": "new_h1_NORMAL_GradientBoosting_N20_t6", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 1, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "US3Y_Rate_ret_5d", "3M_vol_20d", "PCAR_PaccarInc_ret_5d", "Industrial_Production_zscore_60d", "EWQ_France_ret_20d", "US6M_Rate_ret_20d", "US3M_Rate_vol_20d", "XOM_ret_20d", "vix_acceleration_1d", "TGT_Target_zscore_60d", "VVIX_ret_20d", "EWM_Malaysia_zscore_60d", "LMT_LockheedMartin_ret_1d", "HangSeng_HK_ret_1d", "HD_ret_20d", "DHR_ret_1d", "MSTR_Bitcoin3_ret_1d", "T10Y2Y_Spread_ret_5d"], "is_new": true}, {"model_id": "new_h1_NORMAL_GradientBoosting_N20_t7", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 1, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "SPY_zscore_60d", "US7Y_Rate_ret_20d", "EWA_Australia_zscore_60d", "LOW_Lowes_ret_5d", "PCAR_PaccarInc_ret_5d", "CPB_CampbellSoup_ret_5d", "DIS_vol_20d", "NWL_Newell_ret_20d", "NFCI_ret_5d", "CCI_CrownCastle_vol_20d", "CMCSA_ret_1d", "BLK_BlackRock_zscore_60d", "DE_Deere_ret_5d", "QQQ_vol_20d", "EWJ_Japan_vol_20d", "PAYX_Paychex_vol_20d", "Brent_Oil_FRED_ret_5d", "MO_AltriaMG_ret_1d"], "is_new": true}, {"model_id": "new_h1_NORMAL_GradientBoosting_N25_t0", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 1, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "US7Y_Rate_ret_20d", "SJM_JM_Smucker_ret_1d", "heston_ev_h3", "NWL_Newell_ret_20d", "ORCL_zscore_60d", "SBUX_zscore_60d", "NOC_Northrop_ret_20d", "EMR_Emerson_ret_20d", "DHR_vol_20d", "SLB_Schlumberger_ret_1d", "DAX_Germany_zscore_60d", "SBUX_vol_20d", "PFE_ret_1d", "DE_Deere_ret_5d", "SO_SouthernCo_ret_5d", "TED_Spread_zscore_60d", "Core_PCE_zscore_60d", "EWJ_Japan_vol_20d", "PCAR_PaccarInc_ret_5d", "QQQ_vol_20d", "ASX_Australia_ret_5d", "EFFR_vol_20d", "US30Y_Rate_ret_20d"], "is_new": true}, {"model_id": "new_h1_NORMAL_GradientBoosting_N25_t1", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 1, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "Core_PCE_zscore_60d", "MO_AltriaMG_ret_1d", "US3Y_Rate_ret_5d", "HD_zscore_60d", "AVB_AvalonBay_zscore_60d", "T10Y2Y_Spread_ret_5d", "EFFR_ret_1d", "vix_mean_abs_ret_5d", "MRK_Merck_zscore_60d", "HangSeng_HK_vol_20d", "heston_var_ev_h3", "HangSeng_HK_ret_5d", "HUM_Humana_ret_5d", "SBUX_vol_20d", "AORD_AUS_zscore_60d", "DHR_vol_20d", "XLK_Tech_zscore_60d", "VOD_Vodafone_zscore_60d", "NVDA_vol_20d", "AXP_Amex_vol_20d", "heston_ev_h3", "HD_ret_20d", "EWL_Switzerland_vol_20d"], "is_new": true}, {"model_id": "new_h1_NORMAL_GradientBoosting_N25_t2", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 1, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "T_ret_1d", "LMT_LockheedMartin_ret_1d", "BTI_BritishAmerican_ret_5d", "AXP_Amex_ret_20d", "NOC_Northrop_ret_20d", "SBUX_ret_5d", "US3Y_Rate_ret_5d", "IBEX_Spain_ret_20d", "US3M_Rate_vol_20d", "PAYX_Paychex_zscore_60d", "Industrial_Production_zscore_60d", "MSTR_Bitcoin3_ret_1d", "QQQ_vol_20d", "XLF_Fin_vol_20d", "DE_Deere_ret_5d", "DHR_ret_1d", "SLB_Schlumberger_ret_1d", "PFE_ret_1d", "heston_ev_h3", "PAYX_Paychex_vol_20d", "DOW_Price_zscore_60d", "IWM_SmallCap_vol_20d", "CLX_Clorox_vol_20d"], "is_new": true}, {"model_id": "new_h1_NORMAL_GradientBoosting_N25_t3", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 1, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "MS_MorganStanley_zscore_60d", "Industrial_Production_zscore_60d", "TM_Telephone_vol_20d", "AXP_Amex_ret_20d", "PAYX_Paychex_ret_20d", "HD_ret_1d", "heston_var_ev_h7", "CLX_Clorox_vol_20d", "ENB_EnbridgeInc_ret_1d", "AMZN_ret_5d", "US3Y_Rate_ret_5d", "IBEX_Spain_ret_20d", "US30Y_Rate_ret_20d", "DAX_Germany_vol_20d", "XOM_ret_1d", "CI_Cigna_vol_20d", "EWA_Australia_zscore_60d", "spx_vol_5d", "US5Y_Rate_ret_5d", "BA_ret_1d", "US3M_Rate_zscore_60d", "T10Y2Y_Spread_ret_5d", "EWA_Australia_ret_1d"], "is_new": true}, {"model_id": "new_h1_NORMAL_GradientBoosting_N25_t4", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 1, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "AMGN_Amgen_ret_1d", "GILD_Gilead_ret_20d", "EWG_Germany_vol_20d", "EWS_Singapore_ret_5d", "US6M_Rate_ret_20d", "PAYX_Paychex_ret_20d", "CI_Cigna_vol_20d", "TGT_Target_zscore_60d", "XLF_Fin_vol_20d", "ITT_ITTInc_ret_5d", "M_Macys_vol_20d", "VOD_Vodafone_zscore_60d", "AXP_Amex_ret_20d", "SBUX_zscore_60d", "AMT_AmericanTower_ret_1d", "JNJ_ret_1d", "ORCL_vol_20d", "EQR_Equity_ret_1d", "NOC_Northrop_ret_20d", "AMZN_ret_5d", "TED_Spread_zscore_60d", "INTC_ret_5d", "MSTR_Bitcoin3_ret_20d"], "is_new": true}, {"model_id": "new_h1_NORMAL_GradientBoosting_N25_t5", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 1, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "TM_Telephone_vol_20d", "EOG_EOGResources_ret_5d", "HD_ret_5d", "DHR_vol_20d", "EQR_Equity_ret_1d", "EXC_Exelon_ret_1d", "CLX_Clorox_vol_20d", "EFFR_ret_1d", "HangSeng_HK_vol_20d", "NEE_NextEra_ret_20d", "vix_acceleration_1d", "BLK_BlackRock_zscore_60d", "XLV_Health_zscore_60d", "IWM_SmallCap_vol_20d", "spx_abs_ret_max_5d", "AMZN_ret_5d", "BDX_Becton_Dickinson_ret_20d", "HangSeng_HK_ret_1d", "MS_MorganStanley_ret_1d", "CPB_CampbellSoup_ret_5d", "INTC_ret_5d", "spx_momentum_3d", "NFCI_ret_5d"], "is_new": true}, {"model_id": "new_h1_NORMAL_GradientBoosting_N25_t6", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 1, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "BA_ret_1d", "EWG_Germany_ret_20d", "EWQ_France_zscore_60d", "VVIX_ret_20d", "VOD_Vodafone_zscore_60d", "EWQ_France_ret_20d", "SCHW_Schwab_ret_5d", "SLB_Schlumberger_ret_5d", "SLB_Schlumberger_ret_1d", "MSTR_Bitcoin3_ret_5d", "Core_CPI_zscore_60d", "GD_GeneralDynamics_zscore_60d", "BLK_BlackRock_zscore_60d", "INTC_ret_5d", "PG_ret_20d", "US1Y_Rate_ret_20d", "NEE_NextEra_ret_20d", "BTI_BritishAmerican_ret_20d", "SBUX_ret_5d", "AORD_AUS_zscore_60d", "XLF_Fin_vol_20d", "EWM_Malaysia_zscore_60d", "AMGN_Amgen_ret_1d"], "is_new": true}, {"model_id": "new_h1_NORMAL_GradientBoosting_N25_t7", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 1, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "HangSeng_HK_ret_5d", "US3M_Rate_zscore_60d", "NEE_NextEra_ret_20d", "PAYX_Paychex_vol_20d", "CI_Cigna_vol_20d", "heston_ev_h3", "SJM_JM_Smucker_ret_5d", "EWG_Germany_vol_20d", "DHR_vol_20d", "MRK_Merck_zscore_60d", "TM_Telephone_vol_20d", "EOG_EOGResources_ret_5d", "EWS_Singapore_ret_5d", "TM_Telephone_ret_1d", "NWL_Newell_ret_20d", "US1Y_Rate_ret_5d", "PAYX_Paychex_zscore_60d", "AVB_AvalonBay_zscore_60d", "3M_ret_5d", "PG_ret_20d", "HD_zscore_60d", "ENB_EnbridgeInc_ret_1d", "WTI_Oil_FRED_zscore_60d"], "is_new": true}, {"model_id": "new_h1_NORMAL_GradientBoosting_N30_t0", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 1, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "MSTR_Bitcoin3_ret_20d", "LMT_LockheedMartin_ret_1d", "gjr_condvar_h1", "EWJ_Japan_vol_20d", "US7Y_Rate_ret_20d", "SLB_Schlumberger_ret_5d", "HD_ret_20d", "AXP_Amex_ret_20d", "EMR_Emerson_ret_20d", "TGT_Target_zscore_60d", "US3M_Rate_vol_20d", "BTI_BritishAmerican_ret_5d", "HangSeng_HK_ret_1d", "CLX_Clorox_vol_20d", "MS_MorganStanley_zscore_60d", "DAX_Germany_vol_20d", "DHR_ret_1d", "US1Y_Rate_ret_5d", "HD_ret_1d", "Nikkei_Japan_zscore_60d", "EWH_HongKong_ret_5d", "SLB_Schlumberger_ret_1d", "INTC_ret_5d", "JNJ_ret_1d", "EXC_Exelon_zscore_60d", "PAYX_Paychex_zscore_60d", "GE_ret_1d", "EWS_Singapore_ret_5d"], "is_new": true}, {"model_id": "new_h1_NORMAL_GradientBoosting_N30_t1", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 1, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "PFE_ret_1d", "WTI_Oil_FRED_zscore_60d", "IYM_BasicMaterials_ret_20d", "AMGN_Amgen_ret_1d", "DE_Deere_vol_20d", "heston_var_ev_h7", "3M_ret_5d", "XLF_Fin_vol_20d", "QQQ_vol_20d", "3M_vol_20d", "AMD_ret_1d", "EOG_EOGResources_vol_20d", "NFCI_ret_5d", "US1Y_Rate_ret_5d", "SBUX_zscore_60d", "MS_MorganStanley_ret_5d", "TM_Telephone_ret_1d", "CCI_CrownCastle_vol_20d", "Brent_Oil_FRED_ret_5d", "heston_var_ev_h5", "AORD_AUS_zscore_60d", "PAYX_Paychex_ret_20d", "DAX_Germany_vol_20d", "LMT_LockheedMartin_vol_20d", "MSTR_Bitcoin3_ret_1d", "LOW_Lowes_ret_20d", "MS_MorganStanley_ret_1d", "EQIX_Equinix_ret_5d"], "is_new": true}, {"model_id": "new_h1_NORMAL_GradientBoosting_N30_t2", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 1, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "MO_AltriaMG_ret_1d", "EWA_Australia_zscore_60d", "PAYX_Paychex_ret_20d", "AMD_ret_1d", "EWL_Switzerland_zscore_60d", "AMT_AmericanTower_ret_1d", "HD_ret_1d", "LOW_Lowes_ret_20d", "vix_acceleration_1d", "CMCSA_ret_1d", "XLV_Health_zscore_60d", "XOM_ret_20d", "HangSeng_HK_ret_5d", "spx_abs_ret_max_5d", "EWM_Malaysia_zscore_60d", "SCHW_Schwab_ret_5d", "LOW_Lowes_ret_5d", "ASX_Australia_vol_20d", "heston_var_ev_h7", "XLK_Tech_zscore_60d", "EWH_HongKong_ret_5d", "3M_vol_20d", "EWC_Canada_zscore_60d", "MS_MorganStanley_zscore_60d", "CTAS_Cintas_vol_20d", "WTI_Oil_FRED_zscore_60d", "US3M_Rate_zscore_60d", "SJM_JM_Smucker_ret_5d"], "is_new": true}, {"model_id": "new_h1_NORMAL_GradientBoosting_N30_t3", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 1, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "BTI_BritishAmerican_ret_20d", "VOD_Vodafone_zscore_60d", "DAX_Germany_zscore_60d", "vix_mean_abs_ret_5d", "spx_vol_5d", "EXC_Exelon_zscore_60d", "US3M_Rate_zscore_60d", "EWM_Malaysia_vol_20d", "T10Y2Y_Spread_ret_5d", "GILD_Gilead_ret_20d", "EWA_Australia_zscore_60d", "AMT_AmericanTower_ret_1d", "TGT_Target_zscore_60d", "XLV_Health_zscore_60d", "EWY_Korea_zscore_60d", "US30Y_Rate_ret_20d", "PCAR_PaccarInc_ret_5d", "3M_ret_5d", "Nikkei_Japan_vol_20d", "BA_ret_1d", "ITT_ITTInc_ret_5d", "Nikkei_Japan_zscore_60d", "NEE_NextEra_ret_20d", "heston_ev_h3", "DAX_Germany_vol_20d", "SPY_zscore_60d", "PLD_Prologis_ret_5d", "SCHW_Schwab_ret_5d"], "is_new": true}, {"model_id": "new_h1_NORMAL_GradientBoosting_N30_t4", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 1, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "HD_ret_1d", "NVDA_vol_20d", "TGT_Target_zscore_60d", "PAYX_Paychex_vol_20d", "INTC_ret_1d", "3M_vol_20d", "MS_MorganStanley_ret_5d", "EWA_Australia_zscore_60d", "IYR_US_REIT2_zscore_60d", "CCI_CrownCastle_vol_20d", "US3Y_Rate_ret_5d", "AMZN_ret_5d", "EWM_Malaysia_ret_1d", "NWL_Newell_ret_20d", "Brent_Oil_FRED_ret_5d", "EWY_Korea_ret_20d", "EWC_Canada_zscore_60d", "ENB_EnbridgeInc_ret_1d", "IYM_BasicMaterials_ret_20d", "IBEX_Spain_ret_20d", "BTI_BritishAmerican_ret_20d", "CPB_CampbellSoup_ret_20d", "AORD_AUS_zscore_60d", "Retail_Sales_zscore_60d", "Nikkei_Japan_zscore_60d", "XOM_ret_20d", "heston_var_ev_h3", "EWY_Korea_zscore_60d"], "is_new": true}, {"model_id": "new_h1_NORMAL_GradientBoosting_N30_t5", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 1, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "JNJ_ret_1d", "BA_ret_1d", "IYR_US_REIT2_zscore_60d", "T10Y2Y_Spread_ret_5d", "EWQ_France_ret_20d", "Brent_Oil_FRED_ret_20d", "WTI_Oil_FRED_zscore_60d", "XLY_Disc_vol_20d", "ORCL_zscore_60d", "CPB_CampbellSoup_ret_20d", "XOM_ret_20d", "PAYX_Paychex_vol_20d", "US6M_Rate_ret_20d", "DAX_Germany_vol_20d", "CI_Cigna_vol_20d", "BLK_BlackRock_zscore_60d", "ASX_Australia_ret_5d", "MSTR_Bitcoin3_ret_1d", "INTC_ret_1d", "BTI_BritishAmerican_ret_20d", "DAX_Germany_zscore_60d", "XLK_Tech_zscore_60d", "PAYX_Paychex_ret_20d", "BDX_Becton_Dickinson_ret_20d", "HangSeng_HK_ret_5d", "Nikkei_Japan_zscore_60d", "VVIX_ret_20d", "CPB_CampbellSoup_vol_20d"], "is_new": true}, {"model_id": "new_h1_NORMAL_GradientBoosting_N30_t6", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 1, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "MSTR_Bitcoin3_ret_20d", "M_Macys_vol_20d", "CPB_CampbellSoup_ret_20d", "XLY_Disc_vol_20d", "NFCI_ret_5d", "EWM_Malaysia_zscore_60d", "WTI_Oil_FRED_zscore_60d", "XOM_ret_1d", "AMD_ret_1d", "PCAR_PaccarInc_ret_5d", "MSTR_Bitcoin3_ret_5d", "PG_ret_20d", "Nikkei_Japan_vol_20d", "spx_momentum_3d", "CPB_CampbellSoup_ret_5d", "Industrial_Production_zscore_60d", "AMZN_ret_5d", "heston_ev_h3", "SPY_zscore_60d", "IYM_BasicMaterials_ret_20d", "VOD_Vodafone_zscore_60d", "EWA_Australia_zscore_60d", "CPB_CampbellSoup_zscore_60d", "CPB_CampbellSoup_vol_20d", "AMD_ret_5d", "BTI_BritishAmerican_ret_5d", "VRP_ma5", "DAX_Germany_zscore_60d"], "is_new": true}, {"model_id": "new_h1_NORMAL_GradientBoosting_N30_t7", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 1, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "SBUX_ret_5d", "XLY_Disc_vol_20d", "PAYX_Paychex_ret_20d", "ES_Evergy_ret_1d", "EWS_Singapore_ret_5d", "DHR_ret_1d", "AXP_Amex_ret_20d", "BLK_BlackRock_zscore_60d", "DAX_Germany_zscore_60d", "IBEX_Spain_ret_20d", "US7Y_Rate_ret_20d", "ORCL_vol_20d", "SO_SouthernCo_ret_5d", "LOW_Lowes_ret_20d", "heston_ev_h3", "gjr_condvar_h1", "TM_Telephone_vol_20d", "DIS_vol_20d", "XLB_Materials_zscore_60d", "PPL_PPL_ret_1d", "XLV_Health_zscore_60d", "JNJ_ret_1d", "HD_zscore_60d", "MO_AltriaMG_ret_1d", "AMD_ret_1d", "SPY_zscore_60d", "WTI_Oil_FRED_zscore_60d", "BDX_Becton_Dickinson_ret_20d"], "is_new": true}, {"model_id": "new_h1_NORMAL_RandomForest_N5_t0", "algo": "RandomForest", "regime": "NORMAL", "horizon": 1, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWC_Canada_zscore_60d", "EWG_Germany_ret_20d", "heston_var_ev_h3"], "is_new": true}, {"model_id": "new_h1_NORMAL_RandomForest_N5_t1", "algo": "RandomForest", "regime": "NORMAL", "horizon": 1, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "vix_acceleration_1d", "QQQ_vol_20d", "PLD_Prologis_ret_5d"], "is_new": true}, {"model_id": "new_h1_NORMAL_RandomForest_N5_t2", "algo": "RandomForest", "regime": "NORMAL", "horizon": 1, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "NVDA_vol_20d", "IYM_BasicMaterials_ret_20d", "EMR_Emerson_ret_20d"], "is_new": true}, {"model_id": "new_h1_NORMAL_RandomForest_N5_t3", "algo": "RandomForest", "regime": "NORMAL", "horizon": 1, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWL_Switzerland_zscore_60d", "3M_ret_5d", "heston_var_ev_h3"], "is_new": true}, {"model_id": "new_h1_NORMAL_RandomForest_N5_t4", "algo": "RandomForest", "regime": "NORMAL", "horizon": 1, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWQ_France_zscore_60d", "SBUX_ret_5d", "IYM_BasicMaterials_ret_20d"], "is_new": true}, {"model_id": "new_h1_NORMAL_RandomForest_N5_t5", "algo": "RandomForest", "regime": "NORMAL", "horizon": 1, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "ES_Evergy_ret_1d", "3M_vol_20d", "LOW_Lowes_ret_5d"], "is_new": true}, {"model_id": "new_h1_NORMAL_RandomForest_N5_t6", "algo": "RandomForest", "regime": "NORMAL", "horizon": 1, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "US1Y_Rate_ret_5d", "Core_CPI_zscore_60d", "DHR_vol_20d"], "is_new": true}, {"model_id": "new_h1_NORMAL_RandomForest_N5_t7", "algo": "RandomForest", "regime": "NORMAL", "horizon": 1, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "AXP_Amex_ret_20d", "AORD_AUS_zscore_60d", "EWC_Canada_zscore_60d"], "is_new": true}, {"model_id": "new_h1_NORMAL_RandomForest_N8_t0", "algo": "RandomForest", "regime": "NORMAL", "horizon": 1, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "FedFunds_zscore_60d", "EOG_EOGResources_vol_20d", "Industrial_Production_zscore_60d", "US1Y_Rate_ret_20d", "EWS_Singapore_ret_5d", "EQIX_Equinix_ret_5d"], "is_new": true}, {"model_id": "new_h1_NORMAL_RandomForest_N8_t1", "algo": "RandomForest", "regime": "NORMAL", "horizon": 1, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "CPB_CampbellSoup_ret_5d", "AMT_AmericanTower_ret_1d", "DHR_ret_1d", "MSTR_Bitcoin3_ret_20d", "spx_abs_ret_max_5d", "CTAS_Cintas_vol_20d"], "is_new": true}, {"model_id": "new_h1_NORMAL_RandomForest_N8_t2", "algo": "RandomForest", "regime": "NORMAL", "horizon": 1, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWQ_France_zscore_60d", "MO_AltriaMG_ret_1d", "MRK_Merck_zscore_60d", "spx_vol_5d", "EOG_EOGResources_ret_5d", "TED_Spread_vol_20d"], "is_new": true}, {"model_id": "new_h1_NORMAL_RandomForest_N8_t3", "algo": "RandomForest", "regime": "NORMAL", "horizon": 1, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "DHR_vol_20d", "DE_Deere_ret_5d", "EWA_Australia_ret_1d", "EWA_Australia_zscore_60d", "TGT_Target_zscore_60d", "MSTR_Bitcoin3_ret_1d"], "is_new": true}, {"model_id": "new_h1_NORMAL_RandomForest_N8_t4", "algo": "RandomForest", "regime": "NORMAL", "horizon": 1, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "heston_ev_h3", "EWS_Singapore_ret_5d", "AXP_Amex_ret_20d", "XLV_Health_zscore_60d", "VOD_Vodafone_zscore_60d", "EWA_Australia_ret_1d"], "is_new": true}, {"model_id": "new_h1_NORMAL_RandomForest_N8_t5", "algo": "RandomForest", "regime": "NORMAL", "horizon": 1, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWY_Korea_ret_20d", "BA_ret_1d", "gjr_condvar_h1", "HD_ret_1d", "DAX_Germany_zscore_60d", "SLB_Schlumberger_ret_5d"], "is_new": true}, {"model_id": "new_h1_NORMAL_RandomForest_N8_t6", "algo": "RandomForest", "regime": "NORMAL", "horizon": 1, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "PPL_PPL_ret_1d", "spx_vol_5d", "XLV_Health_zscore_60d", "BTI_BritishAmerican_ret_20d", "ENB_EnbridgeInc_ret_1d", "SLB_Schlumberger_ret_5d"], "is_new": true}, {"model_id": "new_h1_NORMAL_RandomForest_N8_t7", "algo": "RandomForest", "regime": "NORMAL", "horizon": 1, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EQIX_Equinix_ret_5d", "ITT_ITTInc_ret_5d", "MRK_Merck_zscore_60d", "TED_Spread_vol_20d", "SBUX_ret_5d", "SCHW_Schwab_ret_5d"], "is_new": true}, {"model_id": "new_h1_NORMAL_RandomForest_N10_t0", "algo": "RandomForest", "regime": "NORMAL", "horizon": 1, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWH_HongKong_ret_5d", "EWL_Switzerland_zscore_60d", "EWG_Germany_ret_20d", "PAYX_Paychex_zscore_60d", "EWQ_France_ret_20d", "MS_MorganStanley_zscore_60d", "EWL_Switzerland_vol_20d", "HangSeng_HK_vol_20d"], "is_new": true}, {"model_id": "new_h1_NORMAL_RandomForest_N10_t1", "algo": "RandomForest", "regime": "NORMAL", "horizon": 1, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EXC_Exelon_ret_1d", "T_ret_1d", "CI_Cigna_vol_20d", "DE_Deere_vol_20d", "HUM_Humana_ret_5d", "PAYX_Paychex_ret_20d", "SPY_zscore_60d", "LMT_LockheedMartin_vol_20d"], "is_new": true}, {"model_id": "new_h1_NORMAL_RandomForest_N10_t2", "algo": "RandomForest", "regime": "NORMAL", "horizon": 1, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "Nikkei_Japan_vol_20d", "PAYX_Paychex_zscore_60d", "PLD_Prologis_ret_5d", "Nikkei_Japan_zscore_60d", "NEE_NextEra_ret_20d", "MO_AltriaMG_ret_1d", "SJM_JM_Smucker_ret_1d", "INTC_ret_5d"], "is_new": true}, {"model_id": "new_h1_NORMAL_RandomForest_N10_t3", "algo": "RandomForest", "regime": "NORMAL", "horizon": 1, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "PAYX_Paychex_ret_20d", "EWM_Malaysia_zscore_60d", "FedFunds_zscore_60d", "heston_var_ev_h7", "Brent_Oil_FRED_ret_5d", "LOW_Lowes_ret_20d", "CI_Cigna_vol_20d", "HUM_Humana_ret_5d"], "is_new": true}, {"model_id": "new_h1_NORMAL_RandomForest_N10_t4", "algo": "RandomForest", "regime": "NORMAL", "horizon": 1, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "CPB_CampbellSoup_ret_5d", "Core_PCE_zscore_60d", "Brent_Oil_FRED_ret_20d", "LOW_Lowes_ret_5d", "ASX_Australia_vol_20d", "TXN_vol_20d", "EWG_Germany_ret_20d", "IYR_US_REIT2_zscore_60d"], "is_new": true}, {"model_id": "new_h1_NORMAL_RandomForest_N10_t5", "algo": "RandomForest", "regime": "NORMAL", "horizon": 1, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "GILD_Gilead_ret_20d", "EWA_Australia_ret_1d", "XLB_Materials_zscore_60d", "DHR_ret_1d", "TXN_vol_20d", "EXC_Exelon_zscore_60d", "AMD_ret_1d", "HangSeng_HK_ret_1d"], "is_new": true}, {"model_id": "new_h1_NORMAL_RandomForest_N10_t6", "algo": "RandomForest", "regime": "NORMAL", "horizon": 1, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "US6M_Rate_ret_20d", "spx_abs_ret_max_5d", "AMD_ret_5d", "PPL_PPL_ret_1d", "SO_SouthernCo_ret_5d", "DOW_Price_zscore_60d", "MS_MorganStanley_zscore_60d", "EWM_Malaysia_vol_20d"], "is_new": true}, {"model_id": "new_h1_NORMAL_RandomForest_N10_t7", "algo": "RandomForest", "regime": "NORMAL", "horizon": 1, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "SPY_zscore_60d", "VOD_Vodafone_zscore_60d", "US1Y_Rate_ret_20d", "PLD_Prologis_ret_5d", "PG_ret_20d", "JNJ_ret_1d", "CI_Cigna_vol_20d", "INTC_ret_1d"], "is_new": true}, {"model_id": "new_h1_NORMAL_RandomForest_N12_t0", "algo": "RandomForest", "regime": "NORMAL", "horizon": 1, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "ORCL_zscore_60d", "PAYX_Paychex_zscore_60d", "US1Y_Rate_ret_5d", "AORD_AUS_zscore_60d", "MRK_Merck_zscore_60d", "DHR_ret_1d", "US5Y_Rate_ret_5d", "DAX_Germany_zscore_60d", "HangSeng_HK_vol_20d", "AMD_ret_1d"], "is_new": true}, {"model_id": "new_h1_NORMAL_RandomForest_N12_t1", "algo": "RandomForest", "regime": "NORMAL", "horizon": 1, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "NVDA_vol_20d", "NWL_Newell_ret_20d", "Michigan_Sentiment_ret_20d", "TM_Telephone_ret_1d", "HangSeng_HK_ret_1d", "IWM_SmallCap_vol_20d", "VOD_Vodafone_zscore_60d", "AMT_AmericanTower_ret_1d", "US3M_Rate_zscore_60d", "ASX_Australia_ret_5d"], "is_new": true}, {"model_id": "new_h1_NORMAL_RandomForest_N12_t2", "algo": "RandomForest", "regime": "NORMAL", "horizon": 1, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "MSTR_Bitcoin3_ret_20d", "EWG_Germany_vol_20d", "MS_MorganStanley_ret_5d", "BTI_BritishAmerican_ret_20d", "TGT_Target_zscore_60d", "MSTR_Bitcoin3_ret_5d", "EWA_Australia_ret_1d", "QQQ_vol_20d", "CPB_CampbellSoup_zscore_60d", "IYM_BasicMaterials_ret_20d"], "is_new": true}, {"model_id": "new_h1_NORMAL_RandomForest_N12_t3", "algo": "RandomForest", "regime": "NORMAL", "horizon": 1, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EOG_EOGResources_ret_5d", "CPB_CampbellSoup_vol_20d", "US1Y_Rate_ret_20d", "EQIX_Equinix_ret_5d", "ASX_Australia_ret_5d", "XLB_Materials_zscore_60d", "SBUX_zscore_60d", "AMZN_ret_5d", "ES_Evergy_ret_1d", "EWC_Canada_zscore_60d"], "is_new": true}, {"model_id": "new_h1_NORMAL_RandomForest_N12_t4", "algo": "RandomForest", "regime": "NORMAL", "horizon": 1, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWM_Malaysia_zscore_60d", "HD_zscore_60d", "spx_momentum_3d", "TED_Spread_vol_20d", "BTI_BritishAmerican_ret_5d", "PCAR_PaccarInc_ret_5d", "BDX_Becton_Dickinson_ret_20d", "MO_AltriaMG_ret_1d", "SPY_zscore_60d", "AMD_ret_1d"], "is_new": true}, {"model_id": "new_h1_NORMAL_RandomForest_N12_t5", "algo": "RandomForest", "regime": "NORMAL", "horizon": 1, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EFFR_vol_20d", "CTAS_Cintas_vol_20d", "DE_Deere_ret_5d", "TM_Telephone_ret_1d", "NVDA_vol_20d", "Retail_Sales_zscore_60d", "BTI_BritishAmerican_ret_5d", "MS_MorganStanley_ret_1d", "EWG_Germany_vol_20d", "IBEX_Spain_ret_20d"], "is_new": true}, {"model_id": "new_h1_NORMAL_RandomForest_N12_t6", "algo": "RandomForest", "regime": "NORMAL", "horizon": 1, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "VOD_Vodafone_zscore_60d", "EWG_Germany_ret_20d", "SJM_JM_Smucker_ret_5d", "PFE_ret_1d", "GILD_Gilead_ret_20d", "3M_vol_20d", "DHR_vol_20d", "PAYX_Paychex_vol_20d", "BTI_BritishAmerican_ret_20d", "M_Macys_vol_20d"], "is_new": true}, {"model_id": "new_h1_NORMAL_RandomForest_N12_t7", "algo": "RandomForest", "regime": "NORMAL", "horizon": 1, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "PFE_ret_1d", "PG_ret_20d", "US5Y_Rate_ret_5d", "AVB_AvalonBay_zscore_60d", "CMCSA_ret_1d", "HD_ret_5d", "heston_var_ev_h7", "PCAR_PaccarInc_ret_5d", "LLY_zscore_60d", "HangSeng_HK_ret_1d"], "is_new": true}, {"model_id": "new_h1_NORMAL_RandomForest_N15_t0", "algo": "RandomForest", "regime": "NORMAL", "horizon": 1, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EQIX_Equinix_ret_5d", "XLY_Disc_vol_20d", "Nikkei_Japan_zscore_60d", "PFE_ret_1d", "MSTR_Bitcoin3_ret_5d", "spx_momentum_3d", "EQR_Equity_ret_1d", "US3M_Rate_zscore_60d", "CPB_CampbellSoup_ret_20d", "MS_MorganStanley_ret_5d", "AXP_Amex_ret_20d", "EWQ_France_zscore_60d", "EFFR_vol_20d"], "is_new": true}, {"model_id": "new_h1_NORMAL_RandomForest_N15_t1", "algo": "RandomForest", "regime": "NORMAL", "horizon": 1, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "SCHW_Schwab_ret_5d", "NFCI_ret_5d", "US1Y_Rate_ret_20d", "DAX_Germany_vol_20d", "MO_AltriaMG_ret_1d", "Brent_Oil_FRED_ret_5d", "CPB_CampbellSoup_vol_20d", "SLB_Schlumberger_ret_1d", "IWM_SmallCap_vol_20d", "NEE_NextEra_ret_20d", "PAYX_Paychex_zscore_60d", "ENB_EnbridgeInc_ret_1d", "EWY_Korea_zscore_60d"], "is_new": true}, {"model_id": "new_h1_NORMAL_RandomForest_N15_t2", "algo": "RandomForest", "regime": "NORMAL", "horizon": 1, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "Brent_Oil_FRED_ret_20d", "heston_var_ev_h3", "PG_ret_20d", "MSTR_Bitcoin3_ret_20d", "JNJ_ret_1d", "M_Macys_vol_20d", "EWA_Australia_zscore_60d", "MS_MorganStanley_zscore_60d", "ASX_Australia_vol_20d", "AMD_ret_1d", "EWY_Korea_zscore_60d", "Core_CPI_zscore_60d", "DE_Deere_vol_20d"], "is_new": true}, {"model_id": "new_h1_NORMAL_RandomForest_N15_t3", "algo": "RandomForest", "regime": "NORMAL", "horizon": 1, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "BTI_BritishAmerican_ret_5d", "CI_Cigna_vol_20d", "T_ret_1d", "spx_momentum_3d", "EWS_Singapore_ret_5d", "ORCL_zscore_60d", "DHR_ret_1d", "PPL_PPL_ret_1d", "HD_ret_1d", "EMR_Emerson_ret_20d", "EOG_EOGResources_ret_5d", "AMGN_Amgen_ret_1d", "XLF_Fin_vol_20d"], "is_new": true}, {"model_id": "new_h1_NORMAL_RandomForest_N15_t4", "algo": "RandomForest", "regime": "NORMAL", "horizon": 1, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "TGT_Target_zscore_60d", "MS_MorganStanley_ret_5d", "EOG_EOGResources_ret_5d", "Brent_Oil_FRED_ret_5d", "vix_mean_abs_ret_5d", "AMD_ret_1d", "GE_ret_1d", "EFFR_ret_1d", "ORCL_zscore_60d", "LOW_Lowes_ret_20d", "EQIX_Equinix_ret_5d", "EWL_Switzerland_vol_20d", "ASX_Australia_ret_5d"], "is_new": true}, {"model_id": "new_h1_NORMAL_RandomForest_N15_t5", "algo": "RandomForest", "regime": "NORMAL", "horizon": 1, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "heston_ev_h3", "T10Y2Y_Spread_ret_5d", "heston_var_ev_h5", "spx_vol_5d", "spx_momentum_3d", "CPB_CampbellSoup_vol_20d", "MS_MorganStanley_ret_1d", "HangSeng_HK_vol_20d", "PAYX_Paychex_ret_20d", "BA_ret_1d", "WTI_Oil_FRED_zscore_60d", "3M_vol_20d", "AXP_Amex_vol_20d"], "is_new": true}, {"model_id": "new_h1_NORMAL_RandomForest_N15_t6", "algo": "RandomForest", "regime": "NORMAL", "horizon": 1, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWQ_France_zscore_60d", "EWG_Germany_ret_20d", "CMCSA_ret_1d", "AMGN_Amgen_ret_1d", "NVDA_vol_20d", "AORD_AUS_zscore_60d", "AMD_ret_5d", "Industrial_Production_zscore_60d", "EWA_Australia_zscore_60d", "vix_mean_abs_ret_5d", "BTI_BritishAmerican_ret_20d", "EWL_Switzerland_zscore_60d", "CPB_CampbellSoup_ret_20d"], "is_new": true}, {"model_id": "new_h1_NORMAL_RandomForest_N15_t7", "algo": "RandomForest", "regime": "NORMAL", "horizon": 1, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "NOC_Northrop_ret_20d", "SO_SouthernCo_ret_5d", "CPB_CampbellSoup_zscore_60d", "IWM_SmallCap_vol_20d", "ITT_ITTInc_ret_5d", "US30Y_Rate_ret_20d", "XOM_ret_20d", "EWA_Australia_ret_1d", "XOM_ret_1d", "EWH_HongKong_ret_5d", "TM_Telephone_vol_20d", "MS_MorganStanley_zscore_60d", "Nikkei_Japan_vol_20d"], "is_new": true}, {"model_id": "new_h1_NORMAL_RandomForest_N20_t0", "algo": "RandomForest", "regime": "NORMAL", "horizon": 1, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "LOW_Lowes_ret_20d", "ITT_ITTInc_ret_5d", "EOG_EOGResources_vol_20d", "AMD_ret_5d", "M_Macys_vol_20d", "spx_abs_ret_max_5d", "INTC_ret_1d", "MS_MorganStanley_ret_1d", "EWJ_Japan_vol_20d", "PFE_ret_1d", "EWL_Switzerland_zscore_60d", "BTI_BritishAmerican_ret_20d", "spx_momentum_3d", "HD_ret_1d", "hmm_p_stress", "XOM_ret_1d", "MO_AltriaMG_ret_1d", "AXP_Amex_ret_20d"], "is_new": true}, {"model_id": "new_h1_NORMAL_RandomForest_N20_t1", "algo": "RandomForest", "regime": "NORMAL", "horizon": 1, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "WTI_Oil_FRED_zscore_60d", "SJM_JM_Smucker_ret_1d", "CPB_CampbellSoup_vol_20d", "TM_Telephone_ret_1d", "US3M_Rate_zscore_60d", "EOG_EOGResources_ret_5d", "EWY_Korea_zscore_60d", "CPB_CampbellSoup_ret_20d", "IWM_SmallCap_vol_20d", "gjr_condvar_h1", "MS_MorganStanley_ret_1d", "DE_Deere_vol_20d", "IYR_US_REIT2_zscore_60d", "EOG_EOGResources_vol_20d", "heston_ev_h3", "INTC_ret_5d", "vix_acceleration_1d", "SBUX_ret_5d"], "is_new": true}, {"model_id": "new_h1_NORMAL_RandomForest_N20_t2", "algo": "RandomForest", "regime": "NORMAL", "horizon": 1, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "MSTR_Bitcoin3_ret_20d", "SLB_Schlumberger_ret_5d", "heston_var_ev_h3", "EWM_Malaysia_zscore_60d", "SJM_JM_Smucker_ret_5d", "VOD_Vodafone_zscore_60d", "AMD_ret_5d", "AMT_AmericanTower_ret_1d", "XLV_Health_zscore_60d", "DAX_Germany_vol_20d", "IBEX_Spain_ret_20d", "EWQ_France_ret_20d", "Nikkei_Japan_vol_20d", "EWS_Singapore_ret_5d", "XLY_Disc_vol_20d", "CCI_CrownCastle_vol_20d", "BLK_BlackRock_zscore_60d", "DHR_ret_1d"], "is_new": true}, {"model_id": "new_h1_NORMAL_RandomForest_N20_t3", "algo": "RandomForest", "regime": "NORMAL", "horizon": 1, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "DOW_Price_zscore_60d", "MSTR_Bitcoin3_ret_1d", "EWG_Germany_vol_20d", "heston_var_ev_h5", "ASX_Australia_vol_20d", "EWQ_France_ret_20d", "WTI_Oil_FRED_zscore_60d", "PAYX_Paychex_ret_20d", "IYM_BasicMaterials_ret_20d", "DE_Deere_vol_20d", "HD_ret_1d", "INTC_ret_5d", "AVB_AvalonBay_zscore_60d", "T10Y2Y_Spread_ret_5d", "NOC_Northrop_ret_20d", "heston_var_ev_h7", "HD_ret_5d", "AMZN_ret_5d"], "is_new": true}, {"model_id": "new_h1_NORMAL_RandomForest_N20_t4", "algo": "RandomForest", "regime": "NORMAL", "horizon": 1, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWG_Germany_vol_20d", "TM_Telephone_vol_20d", "BA_ret_1d", "vix_acceleration_1d", "EWY_Korea_ret_20d", "EWL_Switzerland_zscore_60d", "XLF_Fin_vol_20d", "MSTR_Bitcoin3_ret_20d", "T10Y2Y_Spread_ret_5d", "ITT_ITTInc_ret_5d", "EQR_Equity_ret_1d", "PAYX_Paychex_ret_20d", "gjr_condvar_h1", "3M_vol_20d", "BTI_BritishAmerican_ret_20d", "SBUX_zscore_60d", "BTI_BritishAmerican_ret_5d", "EWA_Australia_zscore_60d"], "is_new": true}, {"model_id": "new_h1_NORMAL_RandomForest_N20_t5", "algo": "RandomForest", "regime": "NORMAL", "horizon": 1, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "CMCSA_ret_1d", "EOG_EOGResources_vol_20d", "VRP_ma5", "T_ret_1d", "EWG_Germany_vol_20d", "EWQ_France_ret_20d", "SLB_Schlumberger_ret_1d", "CCI_CrownCastle_vol_20d", "Nikkei_Japan_vol_20d", "PLD_Prologis_ret_5d", "IBEX_Spain_ret_20d", "ENB_EnbridgeInc_ret_1d", "EWC_Canada_zscore_60d", "EWS_Singapore_ret_5d", "NWL_Newell_ret_20d", "EQIX_Equinix_ret_5d", "XLF_Fin_vol_20d", "EWY_Korea_ret_20d"], "is_new": true}, {"model_id": "new_h1_NORMAL_RandomForest_N20_t6", "algo": "RandomForest", "regime": "NORMAL", "horizon": 1, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "GE_ret_1d", "SBUX_ret_5d", "CCI_CrownCastle_vol_20d", "LOW_Lowes_ret_5d", "MS_MorganStanley_zscore_60d", "MSTR_Bitcoin3_ret_1d", "ENB_EnbridgeInc_ret_1d", "PPL_PPL_ret_1d", "ORCL_zscore_60d", "NWL_Newell_ret_20d", "SLB_Schlumberger_ret_1d", "XLK_Tech_zscore_60d", "Nikkei_Japan_zscore_60d", "EFFR_ret_1d", "AMT_AmericanTower_ret_1d", "MSTR_Bitcoin3_ret_20d", "Nikkei_Japan_vol_20d", "CI_Cigna_vol_20d"], "is_new": true}, {"model_id": "new_h1_NORMAL_RandomForest_N20_t7", "algo": "RandomForest", "regime": "NORMAL", "horizon": 1, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "MSTR_Bitcoin3_ret_20d", "PG_ret_20d", "HangSeng_HK_vol_20d", "PCAR_PaccarInc_ret_5d", "HD_ret_20d", "MSTR_Bitcoin3_ret_1d", "US30Y_Rate_ret_20d", "XLF_Fin_vol_20d", "SBUX_ret_5d", "Brent_Oil_FRED_ret_20d", "IYR_US_REIT2_zscore_60d", "DE_Deere_vol_20d", "PPL_PPL_ret_1d", "EWC_Canada_zscore_60d", "EFFR_ret_1d", "EWA_Australia_ret_1d", "3M_vol_20d", "AMD_ret_5d"], "is_new": true}, {"model_id": "new_h1_NORMAL_RandomForest_N25_t0", "algo": "RandomForest", "regime": "NORMAL", "horizon": 1, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "NOC_Northrop_ret_20d", "MS_MorganStanley_zscore_60d", "MRK_Merck_zscore_60d", "MSTR_Bitcoin3_ret_5d", "SBUX_zscore_60d", "HangSeng_HK_vol_20d", "heston_var_ev_h5", "VOD_Vodafone_zscore_60d", "Brent_Oil_FRED_ret_20d", "NEE_NextEra_ret_20d", "ASX_Australia_vol_20d", "DAX_Germany_vol_20d", "spx_momentum_3d", "US3M_Rate_vol_20d", "EMR_Emerson_ret_20d", "EWA_Australia_zscore_60d", "ORCL_vol_20d", "EXC_Exelon_ret_1d", "LMT_LockheedMartin_ret_1d", "SLB_Schlumberger_ret_5d", "spx_abs_ret_max_5d", "MO_AltriaMG_ret_1d", "BDX_Becton_Dickinson_ret_20d"], "is_new": true}, {"model_id": "new_h1_NORMAL_RandomForest_N25_t1", "algo": "RandomForest", "regime": "NORMAL", "horizon": 1, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWS_Singapore_ret_5d", "Retail_Sales_zscore_60d", "ASX_Australia_ret_5d", "LOW_Lowes_ret_5d", "gjr_condvar_h1", "AORD_AUS_zscore_60d", "heston_var_ev_h7", "DIS_vol_20d", "EXC_Exelon_zscore_60d", "XLB_Materials_zscore_60d", "PCAR_PaccarInc_ret_5d", "heston_var_ev_h5", "EXC_Exelon_ret_1d", "EWY_Korea_ret_20d", "heston_ev_h3", "US6M_Rate_ret_20d", "EMR_Emerson_ret_20d", "AMD_ret_5d", "Core_CPI_zscore_60d", "CMCSA_ret_1d", "MSTR_Bitcoin3_ret_5d", "MS_MorganStanley_zscore_60d", "CCI_CrownCastle_vol_20d"], "is_new": true}, {"model_id": "new_h1_NORMAL_RandomForest_N25_t2", "algo": "RandomForest", "regime": "NORMAL", "horizon": 1, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "Nikkei_Japan_vol_20d", "EWY_Korea_zscore_60d", "HangSeng_HK_ret_5d", "SJM_JM_Smucker_ret_5d", "TGT_Target_zscore_60d", "MSTR_Bitcoin3_ret_20d", "VOD_Vodafone_zscore_60d", "HangSeng_HK_vol_20d", "NFCI_ret_5d", "CLX_Clorox_vol_20d", "CPB_CampbellSoup_zscore_60d", "US3Y_Rate_ret_5d", "LLY_zscore_60d", "DHR_vol_20d", "GD_GeneralDynamics_zscore_60d", "EWH_HongKong_ret_5d", "MS_MorganStanley_zscore_60d", "heston_var_ev_h5", "3M_vol_20d", "EOG_EOGResources_ret_5d", "SBUX_zscore_60d", "3M_ret_5d", "CPB_CampbellSoup_ret_20d"], "is_new": true}, {"model_id": "new_h1_NORMAL_RandomForest_N25_t3", "algo": "RandomForest", "regime": "NORMAL", "horizon": 1, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "Nikkei_Japan_zscore_60d", "EOG_EOGResources_ret_5d", "EQIX_Equinix_ret_5d", "MRK_Merck_zscore_60d", "ES_Evergy_ret_1d", "DE_Deere_vol_20d", "EWC_Canada_zscore_60d", "SBUX_vol_20d", "EQR_Equity_ret_1d", "SJM_JM_Smucker_ret_5d", "CTAS_Cintas_vol_20d", "PAYX_Paychex_zscore_60d", "US3M_Rate_zscore_60d", "EWJ_Japan_vol_20d", "spx_abs_ret_max_5d", "EWG_Germany_ret_20d", "BLK_BlackRock_zscore_60d", "heston_var_ev_h3", "EWA_Australia_zscore_60d", "EXC_Exelon_zscore_60d", "NEE_NextEra_ret_20d", "ORCL_zscore_60d", "CPB_CampbellSoup_zscore_60d"], "is_new": true}, {"model_id": "new_h1_NORMAL_RandomForest_N25_t4", "algo": "RandomForest", "regime": "NORMAL", "horizon": 1, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "CI_Cigna_vol_20d", "MSTR_Bitcoin3_ret_1d", "DAX_Germany_vol_20d", "GILD_Gilead_ret_20d", "BTI_BritishAmerican_ret_20d", "JNJ_ret_1d", "AORD_AUS_zscore_60d", "US7Y_Rate_ret_20d", "EWQ_France_ret_20d", "Retail_Sales_zscore_60d", "SBUX_vol_20d", "CPB_CampbellSoup_ret_5d", "AMD_ret_1d", "BLK_BlackRock_zscore_60d", "GD_GeneralDynamics_zscore_60d", "3M_vol_20d", "heston_var_ev_h5", "AVB_AvalonBay_zscore_60d", "CPB_CampbellSoup_ret_20d", "EWJ_Japan_vol_20d", "T10Y2Y_Spread_ret_5d", "XLK_Tech_zscore_60d", "vix_acceleration_1d"], "is_new": true}, {"model_id": "new_h1_NORMAL_RandomForest_N25_t5", "algo": "RandomForest", "regime": "NORMAL", "horizon": 1, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "Industrial_Production_zscore_60d", "XLB_Materials_zscore_60d", "Nikkei_Japan_vol_20d", "vix_mean_abs_ret_5d", "MO_AltriaMG_ret_1d", "JNJ_ret_1d", "INTC_ret_1d", "EWQ_France_zscore_60d", "LOW_Lowes_ret_5d", "M_Macys_vol_20d", "MSTR_Bitcoin3_ret_1d", "XLK_Tech_zscore_60d", "BDX_Becton_Dickinson_ret_20d", "LOW_Lowes_ret_20d", "IYM_BasicMaterials_ret_20d", "CPB_CampbellSoup_ret_5d", "3M_ret_5d", "PCAR_PaccarInc_ret_5d", "heston_ev_h3", "DHR_ret_1d", "AMZN_ret_5d", "DAX_Germany_vol_20d", "TED_Spread_zscore_60d"], "is_new": true}, {"model_id": "new_h1_NORMAL_RandomForest_N25_t6", "algo": "RandomForest", "regime": "NORMAL", "horizon": 1, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "XLY_Disc_vol_20d", "spx_momentum_3d", "QQQ_vol_20d", "SO_SouthernCo_ret_5d", "NFCI_ret_5d", "EWA_Australia_ret_1d", "3M_ret_5d", "heston_var_ev_h3", "ITT_ITTInc_ret_5d", "AMGN_Amgen_ret_1d", "LOW_Lowes_ret_5d", "AXP_Amex_ret_20d", "IWM_SmallCap_vol_20d", "PAYX_Paychex_ret_20d", "EWM_Malaysia_zscore_60d", "XLF_Fin_vol_20d", "SBUX_vol_20d", "M_Macys_vol_20d", "CCI_CrownCastle_vol_20d", "LMT_LockheedMartin_ret_1d", "Nikkei_Japan_vol_20d", "spx_vol_5d", "CPB_CampbellSoup_ret_5d"], "is_new": true}, {"model_id": "new_h1_NORMAL_RandomForest_N25_t7", "algo": "RandomForest", "regime": "NORMAL", "horizon": 1, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "DHR_vol_20d", "T10Y2Y_Spread_ret_5d", "US3Y_Rate_ret_5d", "SLB_Schlumberger_ret_1d", "ASX_Australia_ret_5d", "DAX_Germany_zscore_60d", "SJM_JM_Smucker_ret_1d", "CLX_Clorox_vol_20d", "HangSeng_HK_ret_5d", "T_ret_1d", "PLD_Prologis_ret_5d", "EOG_EOGResources_ret_5d", "PAYX_Paychex_vol_20d", "Core_CPI_zscore_60d", "SBUX_ret_5d", "TM_Telephone_vol_20d", "EWQ_France_zscore_60d", "EWS_Singapore_ret_5d", "EWJ_Japan_vol_20d", "XOM_ret_20d", "heston_var_ev_h5", "CPB_CampbellSoup_zscore_60d", "ORCL_vol_20d"], "is_new": true}, {"model_id": "new_h1_NORMAL_RandomForest_N30_t0", "algo": "RandomForest", "regime": "NORMAL", "horizon": 1, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "MSTR_Bitcoin3_ret_5d", "HD_ret_1d", "EWL_Switzerland_zscore_60d", "EXC_Exelon_ret_1d", "QQQ_vol_20d", "DAX_Germany_vol_20d", "DE_Deere_vol_20d", "TGT_Target_zscore_60d", "heston_ev_h3", "SBUX_zscore_60d", "CMCSA_ret_1d", "IYR_US_REIT2_zscore_60d", "EWM_Malaysia_ret_1d", "US5Y_Rate_ret_5d", "WTI_Oil_FRED_zscore_60d", "BTI_BritishAmerican_ret_20d", "JNJ_ret_1d", "MSTR_Bitcoin3_ret_1d", "MSTR_Bitcoin3_ret_20d", "US30Y_Rate_ret_20d", "ORCL_vol_20d", "EWM_Malaysia_zscore_60d", "XOM_ret_1d", "EXC_Exelon_zscore_60d", "Core_CPI_zscore_60d", "HD_zscore_60d", "CLX_Clorox_vol_20d", "CTAS_Cintas_vol_20d"], "is_new": true}, {"model_id": "new_h1_NORMAL_RandomForest_N30_t1", "algo": "RandomForest", "regime": "NORMAL", "horizon": 1, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "US3Y_Rate_ret_5d", "MO_AltriaMG_ret_1d", "DAX_Germany_zscore_60d", "INTC_ret_1d", "MSTR_Bitcoin3_ret_20d", "US5Y_Rate_ret_5d", "3M_ret_5d", "IBEX_Spain_ret_20d", "BLK_BlackRock_zscore_60d", "VRP_ma5", "TM_Telephone_ret_1d", "EWA_Australia_ret_1d", "MS_MorganStanley_ret_1d", "NWL_Newell_ret_20d", "INTC_ret_5d", "LUV_SouthwestAir_ret_5d", "spx_vol_5d", "T10Y2Y_Spread_ret_5d", "heston_ev_h3", "EWL_Switzerland_zscore_60d", "CPB_CampbellSoup_zscore_60d", "MS_MorganStanley_zscore_60d", "AMD_ret_1d", "gjr_condvar_h1", "PFE_ret_1d", "spx_momentum_3d", "DIS_vol_20d", "HUM_Humana_ret_5d"], "is_new": true}, {"model_id": "new_h1_NORMAL_RandomForest_N30_t2", "algo": "RandomForest", "regime": "NORMAL", "horizon": 1, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "WTI_Oil_FRED_zscore_60d", "HD_ret_20d", "DIS_vol_20d", "ASX_Australia_ret_5d", "vix_mean_abs_ret_5d", "INTC_ret_5d", "IBEX_Spain_ret_20d", "SJM_JM_Smucker_ret_5d", "DAX_Germany_vol_20d", "GILD_Gilead_ret_20d", "US3Y_Rate_ret_5d", "Retail_Sales_zscore_60d", "EWL_Switzerland_vol_20d", "SCHW_Schwab_ret_5d", "HD_zscore_60d", "MSTR_Bitcoin3_ret_1d", "BLK_BlackRock_zscore_60d", "IYR_US_REIT2_zscore_60d", "EFFR_ret_1d", "EWH_HongKong_ret_5d", "EWM_Malaysia_ret_1d", "BA_ret_1d", "EQIX_Equinix_ret_5d", "SBUX_vol_20d", "MS_MorganStanley_ret_5d", "HangSeng_HK_ret_1d", "EFFR_vol_20d", "AMT_AmericanTower_ret_1d"], "is_new": true}, {"model_id": "new_h1_NORMAL_RandomForest_N30_t3", "algo": "RandomForest", "regime": "NORMAL", "horizon": 1, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "QQQ_vol_20d", "LMT_LockheedMartin_ret_1d", "MS_MorganStanley_ret_5d", "spx_vol_5d", "Brent_Oil_FRED_ret_20d", "TED_Spread_zscore_60d", "HD_ret_20d", "heston_var_ev_h7", "EWA_Australia_zscore_60d", "T_ret_1d", "spx_momentum_3d", "CPB_CampbellSoup_ret_20d", "DE_Deere_ret_5d", "ITT_ITTInc_ret_5d", "US6M_Rate_ret_20d", "BA_ret_1d", "HangSeng_HK_ret_5d", "PFE_ret_1d", "EMR_Emerson_ret_20d", "US5Y_Rate_ret_5d", "DHR_ret_1d", "PAYX_Paychex_ret_20d", "SBUX_ret_5d", "HD_zscore_60d", "DIS_vol_20d", "EWH_HongKong_ret_5d", "EWL_Switzerland_vol_20d", "EWC_Canada_zscore_60d"], "is_new": true}, {"model_id": "new_h1_NORMAL_RandomForest_N30_t4", "algo": "RandomForest", "regime": "NORMAL", "horizon": 1, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "DIS_vol_20d", "MSTR_Bitcoin3_ret_1d", "Nikkei_Japan_vol_20d", "EOG_EOGResources_ret_5d", "BDX_Becton_Dickinson_ret_20d", "CPB_CampbellSoup_ret_20d", "LOW_Lowes_ret_20d", "QQQ_vol_20d", "EQIX_Equinix_ret_5d", "BA_ret_1d", "Core_PCE_zscore_60d", "SO_SouthernCo_ret_5d", "Industrial_Production_zscore_60d", "VRP_ma5", "ORCL_vol_20d", "DOW_Price_zscore_60d", "Michigan_Sentiment_ret_20d", "EWS_Singapore_ret_5d", "ASX_Australia_ret_5d", "gjr_condvar_h1", "EWQ_France_zscore_60d", "MS_MorganStanley_zscore_60d", "SBUX_zscore_60d", "HangSeng_HK_ret_5d", "EOG_EOGResources_vol_20d", "vix_acceleration_1d", "CLX_Clorox_vol_20d", "PG_ret_20d"], "is_new": true}, {"model_id": "new_h1_NORMAL_RandomForest_N30_t5", "algo": "RandomForest", "regime": "NORMAL", "horizon": 1, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "MO_AltriaMG_ret_1d", "AMD_ret_1d", "Core_PCE_zscore_60d", "Core_CPI_zscore_60d", "CI_Cigna_vol_20d", "DAX_Germany_vol_20d", "SBUX_zscore_60d", "heston_var_ev_h5", "SLB_Schlumberger_ret_5d", "EWM_Malaysia_ret_1d", "AMT_AmericanTower_ret_1d", "VOD_Vodafone_zscore_60d", "US3Y_Rate_ret_5d", "XOM_ret_1d", "Retail_Sales_zscore_60d", "GILD_Gilead_ret_20d", "CPB_CampbellSoup_ret_20d", "VRP_ma5", "EQR_Equity_ret_1d", "MSTR_Bitcoin3_ret_5d", "SBUX_ret_5d", "ORCL_vol_20d", "HangSeng_HK_ret_1d", "XLY_Disc_vol_20d", "DAX_Germany_zscore_60d", "CPB_CampbellSoup_ret_5d", "IWM_SmallCap_vol_20d", "BTI_BritishAmerican_ret_20d"], "is_new": true}, {"model_id": "new_h1_NORMAL_RandomForest_N30_t6", "algo": "RandomForest", "regime": "NORMAL", "horizon": 1, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "MS_MorganStanley_ret_5d", "US1Y_Rate_ret_5d", "TM_Telephone_ret_1d", "TED_Spread_zscore_60d", "SBUX_zscore_60d", "BTI_BritishAmerican_ret_20d", "SPY_zscore_60d", "US7Y_Rate_ret_20d", "spx_vol_5d", "DAX_Germany_vol_20d", "EWH_HongKong_ret_5d", "Core_PCE_zscore_60d", "EQR_Equity_ret_1d", "WTI_Oil_FRED_zscore_60d", "XOM_ret_1d", "3M_vol_20d", "vix_acceleration_1d", "PAYX_Paychex_vol_20d", "INTC_ret_1d", "heston_ev_h3", "AVB_AvalonBay_zscore_60d", "DAX_Germany_zscore_60d", "US5Y_Rate_ret_5d", "EWG_Germany_vol_20d", "EMR_Emerson_ret_20d", "CLX_Clorox_vol_20d", "US30Y_Rate_ret_20d", "EXC_Exelon_ret_1d"], "is_new": true}, {"model_id": "new_h1_NORMAL_RandomForest_N30_t7", "algo": "RandomForest", "regime": "NORMAL", "horizon": 1, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "LUV_SouthwestAir_ret_5d", "TXN_vol_20d", "EOG_EOGResources_ret_5d", "NWL_Newell_ret_20d", "3M_ret_5d", "EWY_Korea_ret_20d", "HangSeng_HK_ret_1d", "AXP_Amex_ret_20d", "SPY_zscore_60d", "HUM_Humana_ret_5d", "TGT_Target_zscore_60d", "PAYX_Paychex_ret_20d", "DHR_ret_1d", "US5Y_Rate_ret_5d", "SBUX_zscore_60d", "INTC_ret_1d", "T10Y2Y_Spread_ret_5d", "LMT_LockheedMartin_vol_20d", "TM_Telephone_ret_1d", "SLB_Schlumberger_ret_5d", "EWH_HongKong_ret_5d", "CPB_CampbellSoup_ret_5d", "heston_var_ev_h5", "EWQ_France_ret_20d", "ASX_Australia_vol_20d", "EWA_Australia_ret_1d", "DHR_vol_20d", "heston_var_ev_h7"], "is_new": true}, {"model_id": "new_h1_NORMAL_LogisticRegression_N5_t0", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 1, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "HD_ret_1d", "DAX_Germany_zscore_60d", "XOM_ret_1d"], "is_new": true}, {"model_id": "new_h1_NORMAL_LogisticRegression_N5_t1", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 1, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "MSTR_Bitcoin3_ret_20d", "hmm_p_stress", "EMR_Emerson_ret_20d"], "is_new": true}, {"model_id": "new_h1_NORMAL_LogisticRegression_N5_t2", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 1, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "NWL_Newell_ret_20d", "DHR_vol_20d", "TM_Telephone_ret_1d"], "is_new": true}, {"model_id": "new_h1_NORMAL_LogisticRegression_N5_t3", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 1, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "MRK_Merck_zscore_60d", "US3M_Rate_vol_20d", "spx_momentum_3d"], "is_new": true}, {"model_id": "new_h1_NORMAL_LogisticRegression_N5_t4", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 1, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "INTC_ret_1d", "ORCL_vol_20d", "MSTR_Bitcoin3_ret_20d"], "is_new": true}, {"model_id": "new_h1_NORMAL_LogisticRegression_N5_t5", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 1, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "DAX_Germany_vol_20d", "TGT_Target_zscore_60d", "CTAS_Cintas_vol_20d"], "is_new": true}, {"model_id": "new_h1_NORMAL_LogisticRegression_N5_t6", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 1, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "heston_var_ev_h3", "CLX_Clorox_vol_20d", "LOW_Lowes_ret_5d"], "is_new": true}, {"model_id": "new_h1_NORMAL_LogisticRegression_N5_t7", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 1, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "GILD_Gilead_ret_20d", "AMD_ret_1d", "CPB_CampbellSoup_ret_5d"], "is_new": true}, {"model_id": "new_h1_NORMAL_LogisticRegression_N8_t0", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 1, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "DAX_Germany_vol_20d", "EOG_EOGResources_vol_20d", "TED_Spread_vol_20d", "HangSeng_HK_vol_20d", "VOD_Vodafone_zscore_60d", "SBUX_zscore_60d"], "is_new": true}, {"model_id": "new_h1_NORMAL_LogisticRegression_N8_t1", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 1, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "XLF_Fin_vol_20d", "CPB_CampbellSoup_ret_5d", "AXP_Amex_vol_20d", "EOG_EOGResources_ret_5d", "INTC_ret_5d", "MS_MorganStanley_ret_1d"], "is_new": true}, {"model_id": "new_h1_NORMAL_LogisticRegression_N8_t2", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 1, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "VRP_ma5", "DAX_Germany_zscore_60d", "EXC_Exelon_zscore_60d", "AMD_ret_1d", "SO_SouthernCo_ret_5d", "EFFR_ret_1d"], "is_new": true}, {"model_id": "new_h1_NORMAL_LogisticRegression_N8_t3", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 1, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "US3M_Rate_zscore_60d", "GD_GeneralDynamics_zscore_60d", "BA_ret_1d", "heston_var_ev_h3", "MO_AltriaMG_ret_1d", "EWY_Korea_ret_20d"], "is_new": true}, {"model_id": "new_h1_NORMAL_LogisticRegression_N8_t4", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 1, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "ORCL_zscore_60d", "LMT_LockheedMartin_vol_20d", "PLD_Prologis_ret_5d", "SJM_JM_Smucker_ret_5d", "HD_ret_5d", "GILD_Gilead_ret_20d"], "is_new": true}, {"model_id": "new_h1_NORMAL_LogisticRegression_N8_t5", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 1, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "HangSeng_HK_vol_20d", "NEE_NextEra_ret_20d", "INTC_ret_1d", "CMCSA_ret_1d", "heston_ev_h3", "CCI_CrownCastle_vol_20d"], "is_new": true}, {"model_id": "new_h1_NORMAL_LogisticRegression_N8_t6", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 1, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "US5Y_Rate_ret_5d", "BDX_Becton_Dickinson_ret_20d", "CI_Cigna_vol_20d", "HD_ret_5d", "US1Y_Rate_ret_20d", "SCHW_Schwab_ret_5d"], "is_new": true}, {"model_id": "new_h1_NORMAL_LogisticRegression_N8_t7", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 1, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "SJM_JM_Smucker_ret_1d", "SO_SouthernCo_ret_5d", "TGT_Target_zscore_60d", "DAX_Germany_vol_20d", "Michigan_Sentiment_ret_20d", "AMD_ret_5d"], "is_new": true}, {"model_id": "new_h1_NORMAL_LogisticRegression_N10_t0", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 1, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "TGT_Target_zscore_60d", "LLY_zscore_60d", "EWS_Singapore_ret_5d", "spx_vol_5d", "TED_Spread_zscore_60d", "XLF_Fin_vol_20d", "AVB_AvalonBay_zscore_60d", "PAYX_Paychex_vol_20d"], "is_new": true}, {"model_id": "new_h1_NORMAL_LogisticRegression_N10_t1", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 1, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "LUV_SouthwestAir_ret_5d", "DE_Deere_vol_20d", "SO_SouthernCo_ret_5d", "LMT_LockheedMartin_vol_20d", "BA_ret_1d", "NWL_Newell_ret_20d", "SCHW_Schwab_ret_5d", "MSTR_Bitcoin3_ret_20d"], "is_new": true}, {"model_id": "new_h1_NORMAL_LogisticRegression_N10_t2", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 1, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWY_Korea_ret_20d", "TM_Telephone_vol_20d", "Core_PCE_zscore_60d", "HD_zscore_60d", "HangSeng_HK_ret_1d", "SBUX_ret_5d", "HUM_Humana_ret_5d", "FedFunds_zscore_60d"], "is_new": true}, {"model_id": "new_h1_NORMAL_LogisticRegression_N10_t3", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 1, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "HangSeng_HK_vol_20d", "heston_var_ev_h5", "SO_SouthernCo_ret_5d", "ASX_Australia_vol_20d", "PG_ret_20d", "XLY_Disc_vol_20d", "SBUX_vol_20d", "BDX_Becton_Dickinson_ret_20d"], "is_new": true}, {"model_id": "new_h1_NORMAL_LogisticRegression_N10_t4", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 1, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "HangSeng_HK_ret_1d", "LMT_LockheedMartin_vol_20d", "CPB_CampbellSoup_zscore_60d", "3M_vol_20d", "LMT_LockheedMartin_ret_1d", "ES_Evergy_ret_1d", "M_Macys_vol_20d", "AMD_ret_1d"], "is_new": true}, {"model_id": "new_h1_NORMAL_LogisticRegression_N10_t5", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 1, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "CI_Cigna_vol_20d", "TED_Spread_zscore_60d", "SO_SouthernCo_ret_5d", "EFFR_ret_1d", "NFCI_ret_5d", "DAX_Germany_vol_20d", "SBUX_vol_20d", "3M_vol_20d"], "is_new": true}, {"model_id": "new_h1_NORMAL_LogisticRegression_N10_t6", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 1, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "MS_MorganStanley_zscore_60d", "MRK_Merck_zscore_60d", "HangSeng_HK_ret_1d", "LUV_SouthwestAir_ret_5d", "Nikkei_Japan_vol_20d", "EQR_Equity_ret_1d", "TM_Telephone_ret_1d", "GD_GeneralDynamics_zscore_60d"], "is_new": true}, {"model_id": "new_h1_NORMAL_LogisticRegression_N10_t7", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 1, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "DHR_ret_1d", "LMT_LockheedMartin_vol_20d", "ORCL_vol_20d", "3M_vol_20d", "LOW_Lowes_ret_20d", "EWC_Canada_zscore_60d", "CTAS_Cintas_vol_20d", "EWA_Australia_ret_1d"], "is_new": true}, {"model_id": "new_h1_NORMAL_LogisticRegression_N12_t0", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 1, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "NVDA_vol_20d", "IYM_BasicMaterials_ret_20d", "ASX_Australia_ret_5d", "EMR_Emerson_ret_20d", "vix_acceleration_1d", "EWC_Canada_zscore_60d", "Retail_Sales_zscore_60d", "GD_GeneralDynamics_zscore_60d", "Core_PCE_zscore_60d", "SBUX_zscore_60d"], "is_new": true}, {"model_id": "new_h1_NORMAL_LogisticRegression_N12_t1", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 1, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "M_Macys_vol_20d", "SJM_JM_Smucker_ret_5d", "MS_MorganStanley_ret_1d", "DE_Deere_ret_5d", "CTAS_Cintas_vol_20d", "CPB_CampbellSoup_zscore_60d", "AMT_AmericanTower_ret_1d", "DAX_Germany_zscore_60d", "EFFR_ret_1d", "LLY_zscore_60d"], "is_new": true}, {"model_id": "new_h1_NORMAL_LogisticRegression_N12_t2", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 1, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "JNJ_ret_1d", "Nikkei_Japan_vol_20d", "VVIX_ret_20d", "AMD_ret_1d", "ORCL_vol_20d", "Retail_Sales_zscore_60d", "HD_zscore_60d", "HD_ret_20d", "SBUX_vol_20d", "CI_Cigna_vol_20d"], "is_new": true}, {"model_id": "new_h1_NORMAL_LogisticRegression_N12_t3", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 1, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWY_Korea_ret_20d", "HD_ret_1d", "Core_CPI_zscore_60d", "FedFunds_zscore_60d", "vix_acceleration_1d", "Nikkei_Japan_vol_20d", "EWC_Canada_zscore_60d", "QQQ_vol_20d", "CPB_CampbellSoup_vol_20d", "SJM_JM_Smucker_ret_5d"], "is_new": true}, {"model_id": "new_h1_NORMAL_LogisticRegression_N12_t4", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 1, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWM_Malaysia_vol_20d", "Core_PCE_zscore_60d", "DE_Deere_ret_5d", "M_Macys_vol_20d", "ASX_Australia_vol_20d", "GILD_Gilead_ret_20d", "EWA_Australia_zscore_60d", "DIS_vol_20d", "Nikkei_Japan_vol_20d", "AMGN_Amgen_ret_1d"], "is_new": true}, {"model_id": "new_h1_NORMAL_LogisticRegression_N12_t5", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 1, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "MSTR_Bitcoin3_ret_20d", "NEE_NextEra_ret_20d", "TED_Spread_vol_20d", "HangSeng_HK_vol_20d", "heston_ev_h3", "EWM_Malaysia_zscore_60d", "EWL_Switzerland_zscore_60d", "Industrial_Production_zscore_60d", "Nikkei_Japan_vol_20d", "Retail_Sales_zscore_60d"], "is_new": true}, {"model_id": "new_h1_NORMAL_LogisticRegression_N12_t6", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 1, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "CTAS_Cintas_vol_20d", "GD_GeneralDynamics_zscore_60d", "LUV_SouthwestAir_ret_5d", "HD_ret_1d", "LMT_LockheedMartin_ret_1d", "NVDA_vol_20d", "EWY_Korea_ret_20d", "INTC_ret_5d", "AXP_Amex_vol_20d", "EQR_Equity_ret_1d"], "is_new": true}, {"model_id": "new_h1_NORMAL_LogisticRegression_N12_t7", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 1, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWJ_Japan_vol_20d", "HD_ret_5d", "PPL_PPL_ret_1d", "HD_ret_1d", "XLF_Fin_vol_20d", "CPB_CampbellSoup_ret_5d", "Nikkei_Japan_zscore_60d", "ES_Evergy_ret_1d", "EWY_Korea_zscore_60d", "SBUX_zscore_60d"], "is_new": true}, {"model_id": "new_h1_NORMAL_LogisticRegression_N15_t0", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 1, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "CPB_CampbellSoup_ret_20d", "vix_mean_abs_ret_5d", "AMD_ret_5d", "US1Y_Rate_ret_20d", "DE_Deere_ret_5d", "IYM_BasicMaterials_ret_20d", "LOW_Lowes_ret_5d", "VOD_Vodafone_zscore_60d", "TED_Spread_zscore_60d", "XLB_Materials_zscore_60d", "MRK_Merck_zscore_60d", "EWQ_France_ret_20d", "hmm_p_stress"], "is_new": true}, {"model_id": "new_h1_NORMAL_LogisticRegression_N15_t1", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 1, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "CPB_CampbellSoup_zscore_60d", "DE_Deere_ret_5d", "gjr_condvar_h1", "XLB_Materials_zscore_60d", "Core_PCE_zscore_60d", "AVB_AvalonBay_zscore_60d", "VOD_Vodafone_zscore_60d", "spx_abs_ret_max_5d", "EWA_Australia_ret_1d", "EWA_Australia_zscore_60d", "PAYX_Paychex_vol_20d", "CPB_CampbellSoup_ret_5d", "Brent_Oil_FRED_ret_5d"], "is_new": true}, {"model_id": "new_h1_NORMAL_LogisticRegression_N15_t2", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 1, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWG_Germany_ret_20d", "TGT_Target_zscore_60d", "EFFR_ret_1d", "AXP_Amex_ret_20d", "DAX_Germany_zscore_60d", "EWA_Australia_zscore_60d", "NVDA_vol_20d", "T_ret_1d", "EWY_Korea_zscore_60d", "US30Y_Rate_ret_20d", "LLY_zscore_60d", "LUV_SouthwestAir_ret_5d", "PAYX_Paychex_vol_20d"], "is_new": true}, {"model_id": "new_h1_NORMAL_LogisticRegression_N15_t3", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 1, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "SBUX_ret_5d", "NVDA_vol_20d", "TED_Spread_vol_20d", "SJM_JM_Smucker_ret_5d", "AMT_AmericanTower_ret_1d", "ENB_EnbridgeInc_ret_1d", "WTI_Oil_FRED_zscore_60d", "GILD_Gilead_ret_20d", "DE_Deere_ret_5d", "HD_zscore_60d", "AXP_Amex_ret_20d", "US3M_Rate_vol_20d", "BLK_BlackRock_zscore_60d"], "is_new": true}, {"model_id": "new_h1_NORMAL_LogisticRegression_N15_t4", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 1, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "TGT_Target_zscore_60d", "SO_SouthernCo_ret_5d", "Brent_Oil_FRED_ret_5d", "ENB_EnbridgeInc_ret_1d", "ORCL_zscore_60d", "SLB_Schlumberger_ret_5d", "MSTR_Bitcoin3_ret_5d", "US30Y_Rate_ret_20d", "MS_MorganStanley_ret_1d", "M_Macys_vol_20d", "XLF_Fin_vol_20d", "GE_ret_1d", "EWG_Germany_vol_20d"], "is_new": true}, {"model_id": "new_h1_NORMAL_LogisticRegression_N15_t5", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 1, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "VRP_ma5", "AMD_ret_1d", "SO_SouthernCo_ret_5d", "PCAR_PaccarInc_ret_5d", "SJM_JM_Smucker_ret_5d", "TED_Spread_vol_20d", "EXC_Exelon_zscore_60d", "HangSeng_HK_ret_5d", "EWL_Switzerland_zscore_60d", "HUM_Humana_ret_5d", "GD_GeneralDynamics_zscore_60d", "US1Y_Rate_ret_20d", "Core_PCE_zscore_60d"], "is_new": true}, {"model_id": "new_h1_NORMAL_LogisticRegression_N15_t6", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 1, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "HD_zscore_60d", "AVB_AvalonBay_zscore_60d", "heston_var_ev_h3", "DIS_vol_20d", "HD_ret_1d", "DE_Deere_vol_20d", "GE_ret_1d", "BA_ret_1d", "heston_var_ev_h5", "EWY_Korea_ret_20d", "EWH_HongKong_ret_5d", "TGT_Target_zscore_60d", "Core_CPI_zscore_60d"], "is_new": true}, {"model_id": "new_h1_NORMAL_LogisticRegression_N15_t7", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 1, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "CI_Cigna_vol_20d", "T_ret_1d", "FedFunds_zscore_60d", "DE_Deere_vol_20d", "NEE_NextEra_ret_20d", "XLK_Tech_zscore_60d", "HUM_Humana_ret_5d", "IWM_SmallCap_vol_20d", "BDX_Becton_Dickinson_ret_20d", "XOM_ret_20d", "AMD_ret_1d", "LOW_Lowes_ret_20d", "EWL_Switzerland_zscore_60d"], "is_new": true}, {"model_id": "new_h1_NORMAL_LogisticRegression_N20_t0", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 1, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "AMGN_Amgen_ret_1d", "TM_Telephone_ret_1d", "spx_momentum_3d", "US7Y_Rate_ret_20d", "CPB_CampbellSoup_zscore_60d", "MSTR_Bitcoin3_ret_20d", "spx_abs_ret_max_5d", "EXC_Exelon_ret_1d", "XOM_ret_20d", "heston_ev_h3", "XLB_Materials_zscore_60d", "US1Y_Rate_ret_20d", "EWG_Germany_vol_20d", "PAYX_Paychex_ret_20d", "EWL_Switzerland_zscore_60d", "IWM_SmallCap_vol_20d", "GD_GeneralDynamics_zscore_60d", "US6M_Rate_ret_20d"], "is_new": true}, {"model_id": "new_h1_NORMAL_LogisticRegression_N20_t1", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 1, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWM_Malaysia_zscore_60d", "ORCL_vol_20d", "MO_AltriaMG_ret_1d", "DIS_vol_20d", "Retail_Sales_zscore_60d", "CMCSA_ret_1d", "SCHW_Schwab_ret_5d", "EQR_Equity_ret_1d", "PFE_ret_1d", "ES_Evergy_ret_1d", "EQIX_Equinix_ret_5d", "EWA_Australia_ret_1d", "US3M_Rate_zscore_60d", "SBUX_zscore_60d", "heston_var_ev_h7", "HD_ret_20d", "AMT_AmericanTower_ret_1d", "IBEX_Spain_ret_20d"], "is_new": true}, {"model_id": "new_h1_NORMAL_LogisticRegression_N20_t2", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 1, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "BTI_BritishAmerican_ret_5d", "DHR_vol_20d", "Brent_Oil_FRED_ret_20d", "Nikkei_Japan_vol_20d", "EWS_Singapore_ret_5d", "PCAR_PaccarInc_ret_5d", "heston_var_ev_h7", "Michigan_Sentiment_ret_20d", "CPB_CampbellSoup_ret_5d", "TED_Spread_vol_20d", "DHR_ret_1d", "AMD_ret_5d", "XLB_Materials_zscore_60d", "MSTR_Bitcoin3_ret_1d", "MSTR_Bitcoin3_ret_5d", "AMT_AmericanTower_ret_1d", "CTAS_Cintas_vol_20d", "heston_var_ev_h5"], "is_new": true}, {"model_id": "new_h1_NORMAL_LogisticRegression_N20_t3", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 1, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "GD_GeneralDynamics_zscore_60d", "DHR_vol_20d", "XOM_ret_20d", "WTI_Oil_FRED_zscore_60d", "AMD_ret_5d", "AMD_ret_1d", "CPB_CampbellSoup_zscore_60d", "TM_Telephone_vol_20d", "Industrial_Production_zscore_60d", "CI_Cigna_vol_20d", "EWG_Germany_ret_20d", "DHR_ret_1d", "Retail_Sales_zscore_60d", "US3M_Rate_vol_20d", "PAYX_Paychex_ret_20d", "EMR_Emerson_ret_20d", "HangSeng_HK_vol_20d", "IYM_BasicMaterials_ret_20d"], "is_new": true}, {"model_id": "new_h1_NORMAL_LogisticRegression_N20_t4", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 1, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "TXN_vol_20d", "GD_GeneralDynamics_zscore_60d", "XOM_ret_20d", "DAX_Germany_vol_20d", "AMT_AmericanTower_ret_1d", "AXP_Amex_ret_20d", "CPB_CampbellSoup_vol_20d", "HUM_Humana_ret_5d", "SLB_Schlumberger_ret_5d", "NOC_Northrop_ret_20d", "DHR_vol_20d", "3M_ret_5d", "EWM_Malaysia_ret_1d", "AMZN_ret_5d", "EWM_Malaysia_vol_20d", "Nikkei_Japan_vol_20d", "MSTR_Bitcoin3_ret_5d", "DIS_vol_20d"], "is_new": true}, {"model_id": "new_h1_NORMAL_LogisticRegression_N20_t5", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 1, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "CTAS_Cintas_vol_20d", "SBUX_zscore_60d", "EOG_EOGResources_ret_5d", "EWM_Malaysia_ret_1d", "CPB_CampbellSoup_vol_20d", "HD_ret_20d", "LOW_Lowes_ret_20d", "Brent_Oil_FRED_ret_5d", "XLV_Health_zscore_60d", "IYR_US_REIT2_zscore_60d", "Core_PCE_zscore_60d", "LOW_Lowes_ret_5d", "CI_Cigna_vol_20d", "AXP_Amex_ret_20d", "XLB_Materials_zscore_60d", "DOW_Price_zscore_60d", "EQIX_Equinix_ret_5d", "Nikkei_Japan_vol_20d"], "is_new": true}, {"model_id": "new_h1_NORMAL_LogisticRegression_N20_t6", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 1, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "DOW_Price_zscore_60d", "vix_acceleration_1d", "HangSeng_HK_ret_5d", "XOM_ret_1d", "HD_ret_5d", "LMT_LockheedMartin_ret_1d", "CPB_CampbellSoup_vol_20d", "GE_ret_1d", "SJM_JM_Smucker_ret_1d", "spx_vol_5d", "MO_AltriaMG_ret_1d", "FedFunds_zscore_60d", "Brent_Oil_FRED_ret_5d", "3M_ret_5d", "spx_abs_ret_max_5d", "DIS_vol_20d", "INTC_ret_5d", "MS_MorganStanley_ret_1d"], "is_new": true}, {"model_id": "new_h1_NORMAL_LogisticRegression_N20_t7", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 1, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "US5Y_Rate_ret_5d", "Nikkei_Japan_vol_20d", "HD_ret_1d", "DIS_vol_20d", "CMCSA_ret_1d", "ITT_ITTInc_ret_5d", "DAX_Germany_zscore_60d", "EQR_Equity_ret_1d", "MSTR_Bitcoin3_ret_5d", "LMT_LockheedMartin_vol_20d", "INTC_ret_5d", "SCHW_Schwab_ret_5d", "NWL_Newell_ret_20d", "SBUX_ret_5d", "NFCI_ret_5d", "XLB_Materials_zscore_60d", "heston_ev_h3", "IYR_US_REIT2_zscore_60d"], "is_new": true}, {"model_id": "new_h1_NORMAL_LogisticRegression_N25_t0", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 1, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "T_ret_1d", "SJM_JM_Smucker_ret_1d", "TM_Telephone_ret_1d", "TGT_Target_zscore_60d", "SCHW_Schwab_ret_5d", "spx_momentum_3d", "spx_vol_5d", "EWM_Malaysia_vol_20d", "MSTR_Bitcoin3_ret_5d", "CI_Cigna_vol_20d", "US6M_Rate_ret_20d", "NVDA_vol_20d", "TED_Spread_vol_20d", "US5Y_Rate_ret_5d", "EWL_Switzerland_zscore_60d", "CLX_Clorox_vol_20d", "EWQ_France_ret_20d", "LOW_Lowes_ret_5d", "3M_ret_5d", "EWG_Germany_vol_20d", "AMGN_Amgen_ret_1d", "XLB_Materials_zscore_60d", "US1Y_Rate_ret_5d"], "is_new": true}, {"model_id": "new_h1_NORMAL_LogisticRegression_N25_t1", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 1, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "LMT_LockheedMartin_ret_1d", "LOW_Lowes_ret_5d", "DHR_ret_1d", "HUM_Humana_ret_5d", "PFE_ret_1d", "DHR_vol_20d", "US1Y_Rate_ret_20d", "NVDA_vol_20d", "EWS_Singapore_ret_5d", "EWH_HongKong_ret_5d", "EWM_Malaysia_ret_1d", "BDX_Becton_Dickinson_ret_20d", "spx_vol_5d", "Core_PCE_zscore_60d", "hmm_p_stress", "MSTR_Bitcoin3_ret_1d", "AMZN_ret_5d", "NWL_Newell_ret_20d", "MRK_Merck_zscore_60d", "BA_ret_1d", "CPB_CampbellSoup_zscore_60d", "US6M_Rate_ret_20d", "Brent_Oil_FRED_ret_20d"], "is_new": true}, {"model_id": "new_h1_NORMAL_LogisticRegression_N25_t2", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 1, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "LUV_SouthwestAir_ret_5d", "DHR_vol_20d", "CCI_CrownCastle_vol_20d", "EWS_Singapore_ret_5d", "Brent_Oil_FRED_ret_5d", "AMD_ret_5d", "US3M_Rate_zscore_60d", "NOC_Northrop_ret_20d", "DOW_Price_zscore_60d", "EQR_Equity_ret_1d", "PPL_PPL_ret_1d", "DHR_ret_1d", "TED_Spread_zscore_60d", "HangSeng_HK_ret_1d", "gjr_condvar_h1", "NFCI_ret_5d", "Industrial_Production_zscore_60d", "PAYX_Paychex_ret_20d", "hmm_p_stress", "HD_ret_5d", "CTAS_Cintas_vol_20d", "EFFR_ret_1d", "BLK_BlackRock_zscore_60d"], "is_new": true}, {"model_id": "new_h1_NORMAL_LogisticRegression_N25_t3", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 1, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "AMD_ret_1d", "MS_MorganStanley_zscore_60d", "NWL_Newell_ret_20d", "PCAR_PaccarInc_ret_5d", "US1Y_Rate_ret_20d", "EXC_Exelon_zscore_60d", "US3Y_Rate_ret_5d", "SJM_JM_Smucker_ret_5d", "HangSeng_HK_vol_20d", "EWM_Malaysia_vol_20d", "spx_vol_5d", "EFFR_vol_20d", "GE_ret_1d", "CMCSA_ret_1d", "T10Y2Y_Spread_ret_5d", "MSTR_Bitcoin3_ret_1d", "DAX_Germany_zscore_60d", "EWJ_Japan_vol_20d", "M_Macys_vol_20d", "BDX_Becton_Dickinson_ret_20d", "EWL_Switzerland_zscore_60d", "MS_MorganStanley_ret_5d", "ES_Evergy_ret_1d"], "is_new": true}, {"model_id": "new_h1_NORMAL_LogisticRegression_N25_t4", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 1, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "INTC_ret_1d", "spx_abs_ret_max_5d", "heston_var_ev_h5", "Michigan_Sentiment_ret_20d", "MSTR_Bitcoin3_ret_20d", "ENB_EnbridgeInc_ret_1d", "BA_ret_1d", "MRK_Merck_zscore_60d", "NWL_Newell_ret_20d", "EWA_Australia_zscore_60d", "GE_ret_1d", "CPB_CampbellSoup_zscore_60d", "CPB_CampbellSoup_vol_20d", "Brent_Oil_FRED_ret_5d", "LOW_Lowes_ret_5d", "XLB_Materials_zscore_60d", "IYR_US_REIT2_zscore_60d", "AORD_AUS_zscore_60d", "EWQ_France_zscore_60d", "HD_ret_20d", "EMR_Emerson_ret_20d", "TM_Telephone_vol_20d", "XOM_ret_20d"], "is_new": true}, {"model_id": "new_h1_NORMAL_LogisticRegression_N25_t5", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 1, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWA_Australia_ret_1d", "US1Y_Rate_ret_20d", "EWG_Germany_ret_20d", "heston_var_ev_h3", "ASX_Australia_ret_5d", "EWQ_France_zscore_60d", "GD_GeneralDynamics_zscore_60d", "CI_Cigna_vol_20d", "US3M_Rate_zscore_60d", "US3M_Rate_vol_20d", "AXP_Amex_ret_20d", "heston_ev_h3", "SBUX_ret_5d", "gjr_condvar_h1", "AXP_Amex_vol_20d", "INTC_ret_1d", "CMCSA_ret_1d", "IBEX_Spain_ret_20d", "AMD_ret_1d", "LOW_Lowes_ret_20d", "HD_ret_20d", "MS_MorganStanley_zscore_60d", "SJM_JM_Smucker_ret_5d"], "is_new": true}, {"model_id": "new_h1_NORMAL_LogisticRegression_N25_t6", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 1, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "DE_Deere_ret_5d", "CPB_CampbellSoup_ret_20d", "IYM_BasicMaterials_ret_20d", "NEE_NextEra_ret_20d", "SJM_JM_Smucker_ret_5d", "CPB_CampbellSoup_ret_5d", "spx_abs_ret_max_5d", "DOW_Price_zscore_60d", "AMZN_ret_5d", "VOD_Vodafone_zscore_60d", "MSTR_Bitcoin3_ret_20d", "Retail_Sales_zscore_60d", "heston_ev_h3", "ASX_Australia_vol_20d", "HD_zscore_60d", "EMR_Emerson_ret_20d", "heston_var_ev_h5", "PG_ret_20d", "CCI_CrownCastle_vol_20d", "CMCSA_ret_1d", "EWM_Malaysia_zscore_60d", "EWL_Switzerland_vol_20d", "DHR_vol_20d"], "is_new": true}, {"model_id": "new_h1_NORMAL_LogisticRegression_N25_t7", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 1, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "DE_Deere_ret_5d", "Industrial_Production_zscore_60d", "HD_zscore_60d", "LOW_Lowes_ret_5d", "XOM_ret_1d", "Nikkei_Japan_zscore_60d", "AVB_AvalonBay_zscore_60d", "MS_MorganStanley_ret_1d", "EWQ_France_zscore_60d", "EWH_HongKong_ret_5d", "XLK_Tech_zscore_60d", "XLF_Fin_vol_20d", "Brent_Oil_FRED_ret_5d", "EFFR_ret_1d", "US3M_Rate_zscore_60d", "SJM_JM_Smucker_ret_1d", "CPB_CampbellSoup_ret_20d", "AMD_ret_5d", "DHR_vol_20d", "IBEX_Spain_ret_20d", "T_ret_1d", "CLX_Clorox_vol_20d", "gjr_condvar_h1"], "is_new": true}, {"model_id": "new_h1_NORMAL_LogisticRegression_N30_t0", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 1, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "HUM_Humana_ret_5d", "EMR_Emerson_ret_20d", "BTI_BritishAmerican_ret_5d", "HD_ret_1d", "US5Y_Rate_ret_5d", "LUV_SouthwestAir_ret_5d", "LMT_LockheedMartin_vol_20d", "CLX_Clorox_vol_20d", "Brent_Oil_FRED_ret_20d", "EQIX_Equinix_ret_5d", "PFE_ret_1d", "EXC_Exelon_zscore_60d", "QQQ_vol_20d", "EXC_Exelon_ret_1d", "PAYX_Paychex_ret_20d", "DHR_ret_1d", "DAX_Germany_vol_20d", "US3M_Rate_zscore_60d", "SBUX_vol_20d", "AVB_AvalonBay_zscore_60d", "hmm_p_stress", "HD_ret_5d", "M_Macys_vol_20d", "NEE_NextEra_ret_20d", "DE_Deere_ret_5d", "ORCL_zscore_60d", "VRP_ma5", "SO_SouthernCo_ret_5d"], "is_new": true}, {"model_id": "new_h1_NORMAL_LogisticRegression_N30_t1", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 1, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "NFCI_ret_5d", "SLB_Schlumberger_ret_1d", "PAYX_Paychex_zscore_60d", "FedFunds_zscore_60d", "AVB_AvalonBay_zscore_60d", "XLY_Disc_vol_20d", "heston_var_ev_h5", "BDX_Becton_Dickinson_ret_20d", "EWS_Singapore_ret_5d", "US30Y_Rate_ret_20d", "AMD_ret_1d", "EFFR_ret_1d", "CPB_CampbellSoup_vol_20d", "HD_zscore_60d", "MSTR_Bitcoin3_ret_1d", "EFFR_vol_20d", "US3M_Rate_zscore_60d", "M_Macys_vol_20d", "Brent_Oil_FRED_ret_20d", "EWM_Malaysia_ret_1d", "MS_MorganStanley_ret_1d", "PG_ret_20d", "CMCSA_ret_1d", "LMT_LockheedMartin_ret_1d", "PLD_Prologis_ret_5d", "US5Y_Rate_ret_5d", "TGT_Target_zscore_60d", "US3Y_Rate_ret_5d"], "is_new": true}, {"model_id": "new_h1_NORMAL_LogisticRegression_N30_t2", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 1, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "MS_MorganStanley_zscore_60d", "Nikkei_Japan_zscore_60d", "EWG_Germany_ret_20d", "M_Macys_vol_20d", "IYM_BasicMaterials_ret_20d", "BTI_BritishAmerican_ret_20d", "INTC_ret_5d", "JNJ_ret_1d", "XLB_Materials_zscore_60d", "SBUX_vol_20d", "EWJ_Japan_vol_20d", "QQQ_vol_20d", "Core_CPI_zscore_60d", "HD_ret_1d", "PCAR_PaccarInc_ret_5d", "ASX_Australia_vol_20d", "TXN_vol_20d", "EFFR_ret_1d", "PPL_PPL_ret_1d", "DIS_vol_20d", "gjr_condvar_h1", "heston_var_ev_h7", "US3M_Rate_zscore_60d", "ASX_Australia_ret_5d", "MSTR_Bitcoin3_ret_20d", "BLK_BlackRock_zscore_60d", "LOW_Lowes_ret_5d", "vix_acceleration_1d"], "is_new": true}, {"model_id": "new_h1_NORMAL_LogisticRegression_N30_t3", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 1, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "AMT_AmericanTower_ret_1d", "NWL_Newell_ret_20d", "GE_ret_1d", "EWY_Korea_zscore_60d", "heston_ev_h3", "EQIX_Equinix_ret_5d", "CI_Cigna_vol_20d", "DHR_vol_20d", "BTI_BritishAmerican_ret_5d", "GILD_Gilead_ret_20d", "Michigan_Sentiment_ret_20d", "CCI_CrownCastle_vol_20d", "HangSeng_HK_ret_1d", "PAYX_Paychex_zscore_60d", "NVDA_vol_20d", "TED_Spread_vol_20d", "EWJ_Japan_vol_20d", "EXC_Exelon_zscore_60d", "ES_Evergy_ret_1d", "BA_ret_1d", "CMCSA_ret_1d", "Nikkei_Japan_vol_20d", "JNJ_ret_1d", "XOM_ret_1d", "ENB_EnbridgeInc_ret_1d", "EWC_Canada_zscore_60d", "LMT_LockheedMartin_ret_1d", "XLV_Health_zscore_60d"], "is_new": true}, {"model_id": "new_h1_NORMAL_LogisticRegression_N30_t4", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 1, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "LMT_LockheedMartin_vol_20d", "US3Y_Rate_ret_5d", "MS_MorganStanley_ret_1d", "AMD_ret_1d", "US30Y_Rate_ret_20d", "VVIX_ret_20d", "EWC_Canada_zscore_60d", "Brent_Oil_FRED_ret_5d", "US6M_Rate_ret_20d", "XLV_Health_zscore_60d", "DIS_vol_20d", "ITT_ITTInc_ret_5d", "INTC_ret_5d", "AMD_ret_5d", "DOW_Price_zscore_60d", "PFE_ret_1d", "gjr_condvar_h1", "SJM_JM_Smucker_ret_1d", "XOM_ret_1d", "Core_CPI_zscore_60d", "NEE_NextEra_ret_20d", "FedFunds_zscore_60d", "T10Y2Y_Spread_ret_5d", "3M_ret_5d", "PAYX_Paychex_zscore_60d", "DAX_Germany_zscore_60d", "US5Y_Rate_ret_5d", "CPB_CampbellSoup_ret_20d"], "is_new": true}, {"model_id": "new_h1_NORMAL_LogisticRegression_N30_t5", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 1, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "TXN_vol_20d", "ENB_EnbridgeInc_ret_1d", "WTI_Oil_FRED_zscore_60d", "MSTR_Bitcoin3_ret_1d", "EWL_Switzerland_vol_20d", "LMT_LockheedMartin_ret_1d", "EWH_HongKong_ret_5d", "EWA_Australia_ret_1d", "US5Y_Rate_ret_5d", "AMD_ret_5d", "MO_AltriaMG_ret_1d", "LUV_SouthwestAir_ret_5d", "AMGN_Amgen_ret_1d", "HangSeng_HK_ret_5d", "HangSeng_HK_vol_20d", "SJM_JM_Smucker_ret_5d", "DAX_Germany_vol_20d", "SJM_JM_Smucker_ret_1d", "PCAR_PaccarInc_ret_5d", "NFCI_ret_5d", "VVIX_ret_20d", "LMT_LockheedMartin_vol_20d", "VOD_Vodafone_zscore_60d", "TM_Telephone_vol_20d", "Nikkei_Japan_vol_20d", "T10Y2Y_Spread_ret_5d", "IYR_US_REIT2_zscore_60d", "Core_CPI_zscore_60d"], "is_new": true}, {"model_id": "new_h1_NORMAL_LogisticRegression_N30_t6", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 1, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EQIX_Equinix_ret_5d", "US1Y_Rate_ret_20d", "VVIX_ret_20d", "EMR_Emerson_ret_20d", "TM_Telephone_vol_20d", "ORCL_vol_20d", "MSTR_Bitcoin3_ret_20d", "HD_ret_20d", "hmm_p_stress", "heston_ev_h3", "heston_var_ev_h5", "XLY_Disc_vol_20d", "CPB_CampbellSoup_ret_5d", "CLX_Clorox_vol_20d", "CPB_CampbellSoup_vol_20d", "XOM_ret_20d", "AMGN_Amgen_ret_1d", "NFCI_ret_5d", "XLV_Health_zscore_60d", "SJM_JM_Smucker_ret_1d", "TXN_vol_20d", "XOM_ret_1d", "PAYX_Paychex_ret_20d", "SJM_JM_Smucker_ret_5d", "ASX_Australia_ret_5d", "HD_zscore_60d", "NVDA_vol_20d", "EWJ_Japan_vol_20d"], "is_new": true}, {"model_id": "new_h1_NORMAL_LogisticRegression_N30_t7", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 1, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWC_Canada_zscore_60d", "PPL_PPL_ret_1d", "DAX_Germany_vol_20d", "LMT_LockheedMartin_ret_1d", "PG_ret_20d", "NFCI_ret_5d", "heston_var_ev_h7", "MS_MorganStanley_zscore_60d", "3M_vol_20d", "MRK_Merck_zscore_60d", "EWM_Malaysia_ret_1d", "SLB_Schlumberger_ret_1d", "HD_ret_20d", "GD_GeneralDynamics_zscore_60d", "FedFunds_zscore_60d", "HangSeng_HK_vol_20d", "NOC_Northrop_ret_20d", "EWQ_France_zscore_60d", "heston_var_ev_h3", "XLF_Fin_vol_20d", "ASX_Australia_ret_5d", "AMD_ret_5d", "AMZN_ret_5d", "EFFR_ret_1d", "heston_ev_h3", "SPY_zscore_60d", "INTC_ret_5d", "IYR_US_REIT2_zscore_60d"], "is_new": true}, {"model_id": "new_h1_STRESS_XGBoost_N5_t0", "algo": "XGBoost", "regime": "STRESS", "horizon": 1, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWA_Australia_zscore_60d", "XLV_Health_zscore_60d", "heston_var_ev_h5"], "is_new": true}, {"model_id": "new_h1_STRESS_XGBoost_N5_t1", "algo": "XGBoost", "regime": "STRESS", "horizon": 1, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "heston_ev_h3", "MS_MorganStanley_ret_1d", "EOG_EOGResources_vol_20d"], "is_new": true}, {"model_id": "new_h1_STRESS_XGBoost_N5_t2", "algo": "XGBoost", "regime": "STRESS", "horizon": 1, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWM_Malaysia_zscore_60d", "CPB_CampbellSoup_ret_5d", "DAX_Germany_zscore_60d"], "is_new": true}, {"model_id": "new_h1_STRESS_XGBoost_N5_t3", "algo": "XGBoost", "regime": "STRESS", "horizon": 1, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "US1Y_Rate_ret_20d", "EFFR_vol_20d", "ITT_ITTInc_ret_5d"], "is_new": true}, {"model_id": "new_h1_STRESS_XGBoost_N5_t4", "algo": "XGBoost", "regime": "STRESS", "horizon": 1, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "ORCL_vol_20d", "AMD_ret_1d", "XOM_ret_20d"], "is_new": true}, {"model_id": "new_h1_STRESS_XGBoost_N5_t5", "algo": "XGBoost", "regime": "STRESS", "horizon": 1, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "US3Y_Rate_ret_5d", "MSTR_Bitcoin3_ret_20d", "AMZN_ret_5d"], "is_new": true}, {"model_id": "new_h1_STRESS_XGBoost_N5_t6", "algo": "XGBoost", "regime": "STRESS", "horizon": 1, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "ES_Evergy_ret_1d", "EWQ_France_zscore_60d", "spx_abs_ret_max_5d"], "is_new": true}, {"model_id": "new_h1_STRESS_XGBoost_N5_t7", "algo": "XGBoost", "regime": "STRESS", "horizon": 1, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "MSTR_Bitcoin3_ret_1d", "EXC_Exelon_zscore_60d", "spx_momentum_3d"], "is_new": true}, {"model_id": "new_h1_STRESS_XGBoost_N8_t0", "algo": "XGBoost", "regime": "STRESS", "horizon": 1, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "XOM_ret_20d", "AMT_AmericanTower_ret_1d", "heston_var_ev_h3", "US6M_Rate_ret_20d", "CPB_CampbellSoup_vol_20d", "CTAS_Cintas_vol_20d"], "is_new": true}, {"model_id": "new_h1_STRESS_XGBoost_N8_t1", "algo": "XGBoost", "regime": "STRESS", "horizon": 1, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "HD_ret_20d", "FedFunds_zscore_60d", "EQR_Equity_ret_1d", "CMCSA_ret_1d", "Brent_Oil_FRED_ret_5d", "SCHW_Schwab_ret_5d"], "is_new": true}, {"model_id": "new_h1_STRESS_XGBoost_N8_t2", "algo": "XGBoost", "regime": "STRESS", "horizon": 1, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "SBUX_zscore_60d", "AMT_AmericanTower_ret_1d", "NFCI_ret_5d", "Brent_Oil_FRED_ret_20d", "SBUX_vol_20d", "XOM_ret_20d"], "is_new": true}, {"model_id": "new_h1_STRESS_XGBoost_N8_t3", "algo": "XGBoost", "regime": "STRESS", "horizon": 1, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "INTC_ret_1d", "heston_ev_h3", "PAYX_Paychex_zscore_60d", "EMR_Emerson_ret_20d", "EQR_Equity_ret_1d", "HangSeng_HK_ret_1d"], "is_new": true}, {"model_id": "new_h1_STRESS_XGBoost_N8_t4", "algo": "XGBoost", "regime": "STRESS", "horizon": 1, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "JNJ_ret_1d", "LLY_zscore_60d", "SBUX_ret_5d", "XLV_Health_zscore_60d", "WTI_Oil_FRED_zscore_60d", "TXN_vol_20d"], "is_new": true}, {"model_id": "new_h1_STRESS_XGBoost_N8_t5", "algo": "XGBoost", "regime": "STRESS", "horizon": 1, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWA_Australia_zscore_60d", "XLF_Fin_vol_20d", "AORD_AUS_zscore_60d", "TM_Telephone_ret_1d", "AMT_AmericanTower_ret_1d", "SPY_zscore_60d"], "is_new": true}, {"model_id": "new_h1_STRESS_XGBoost_N8_t6", "algo": "XGBoost", "regime": "STRESS", "horizon": 1, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "Core_CPI_zscore_60d", "EWL_Switzerland_vol_20d", "AORD_AUS_zscore_60d", "IWM_SmallCap_vol_20d", "DHR_ret_1d", "EWL_Switzerland_zscore_60d"], "is_new": true}, {"model_id": "new_h1_STRESS_XGBoost_N8_t7", "algo": "XGBoost", "regime": "STRESS", "horizon": 1, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "SBUX_ret_5d", "EWG_Germany_ret_20d", "CMCSA_ret_1d", "LMT_LockheedMartin_ret_1d", "US1Y_Rate_ret_20d", "hmm_p_stress"], "is_new": true}, {"model_id": "new_h1_STRESS_XGBoost_N10_t0", "algo": "XGBoost", "regime": "STRESS", "horizon": 1, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "SO_SouthernCo_ret_5d", "M_Macys_vol_20d", "EWA_Australia_zscore_60d", "EWY_Korea_zscore_60d", "EOG_EOGResources_vol_20d", "EWC_Canada_zscore_60d", "HD_ret_1d", "MS_MorganStanley_ret_5d"], "is_new": true}, {"model_id": "new_h1_STRESS_XGBoost_N10_t1", "algo": "XGBoost", "regime": "STRESS", "horizon": 1, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "SBUX_vol_20d", "Nikkei_Japan_zscore_60d", "PAYX_Paychex_vol_20d", "NWL_Newell_ret_20d", "VVIX_ret_20d", "GE_ret_1d", "INTC_ret_5d", "XOM_ret_1d"], "is_new": true}, {"model_id": "new_h1_STRESS_XGBoost_N10_t2", "algo": "XGBoost", "regime": "STRESS", "horizon": 1, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "US3M_Rate_zscore_60d", "LMT_LockheedMartin_vol_20d", "MRK_Merck_zscore_60d", "gjr_condvar_h1", "BTI_BritishAmerican_ret_20d", "PAYX_Paychex_zscore_60d", "ENB_EnbridgeInc_ret_1d", "ASX_Australia_ret_5d"], "is_new": true}, {"model_id": "new_h1_STRESS_XGBoost_N10_t3", "algo": "XGBoost", "regime": "STRESS", "horizon": 1, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "XLB_Materials_zscore_60d", "heston_var_ev_h7", "TED_Spread_zscore_60d", "LMT_LockheedMartin_vol_20d", "MS_MorganStanley_zscore_60d", "AORD_AUS_zscore_60d", "Core_CPI_zscore_60d", "EQR_Equity_ret_1d"], "is_new": true}, {"model_id": "new_h1_STRESS_XGBoost_N10_t4", "algo": "XGBoost", "regime": "STRESS", "horizon": 1, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWS_Singapore_ret_5d", "Brent_Oil_FRED_ret_20d", "EWL_Switzerland_vol_20d", "spx_momentum_3d", "MRK_Merck_zscore_60d", "DIS_vol_20d", "INTC_ret_1d", "DAX_Germany_vol_20d"], "is_new": true}, {"model_id": "new_h1_STRESS_XGBoost_N10_t5", "algo": "XGBoost", "regime": "STRESS", "horizon": 1, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWJ_Japan_vol_20d", "PLD_Prologis_ret_5d", "US3M_Rate_zscore_60d", "ITT_ITTInc_ret_5d", "EWM_Malaysia_zscore_60d", "US30Y_Rate_ret_20d", "BTI_BritishAmerican_ret_20d", "MSTR_Bitcoin3_ret_5d"], "is_new": true}, {"model_id": "new_h1_STRESS_XGBoost_N10_t6", "algo": "XGBoost", "regime": "STRESS", "horizon": 1, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "US30Y_Rate_ret_20d", "HD_ret_1d", "XLF_Fin_vol_20d", "vix_mean_abs_ret_5d", "Brent_Oil_FRED_ret_5d", "US3Y_Rate_ret_5d", "heston_var_ev_h7", "NVDA_vol_20d"], "is_new": true}, {"model_id": "new_h1_STRESS_XGBoost_N10_t7", "algo": "XGBoost", "regime": "STRESS", "horizon": 1, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWC_Canada_zscore_60d", "AXP_Amex_vol_20d", "SCHW_Schwab_ret_5d", "SJM_JM_Smucker_ret_1d", "AXP_Amex_ret_20d", "GILD_Gilead_ret_20d", "NVDA_vol_20d", "Core_PCE_zscore_60d"], "is_new": true}, {"model_id": "new_h1_STRESS_XGBoost_N12_t0", "algo": "XGBoost", "regime": "STRESS", "horizon": 1, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "VRP_ma5", "MSTR_Bitcoin3_ret_5d", "EWQ_France_ret_20d", "US5Y_Rate_ret_5d", "AMT_AmericanTower_ret_1d", "HD_ret_20d", "AMGN_Amgen_ret_1d", "heston_var_ev_h5", "3M_vol_20d", "PG_ret_20d"], "is_new": true}, {"model_id": "new_h1_STRESS_XGBoost_N12_t1", "algo": "XGBoost", "regime": "STRESS", "horizon": 1, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "TGT_Target_zscore_60d", "TED_Spread_zscore_60d", "AVB_AvalonBay_zscore_60d", "Industrial_Production_zscore_60d", "heston_var_ev_h7", "AMGN_Amgen_ret_1d", "BTI_BritishAmerican_ret_5d", "PPL_PPL_ret_1d", "CPB_CampbellSoup_ret_20d", "NOC_Northrop_ret_20d"], "is_new": true}, {"model_id": "new_h1_STRESS_XGBoost_N12_t2", "algo": "XGBoost", "regime": "STRESS", "horizon": 1, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "LLY_zscore_60d", "M_Macys_vol_20d", "CCI_CrownCastle_vol_20d", "US1Y_Rate_ret_5d", "INTC_ret_1d", "SCHW_Schwab_ret_5d", "XLK_Tech_zscore_60d", "AXP_Amex_vol_20d", "MS_MorganStanley_ret_5d", "HangSeng_HK_ret_1d"], "is_new": true}, {"model_id": "new_h1_STRESS_XGBoost_N12_t3", "algo": "XGBoost", "regime": "STRESS", "horizon": 1, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "MRK_Merck_zscore_60d", "XLY_Disc_vol_20d", "CPB_CampbellSoup_vol_20d", "AORD_AUS_zscore_60d", "EWM_Malaysia_ret_1d", "AXP_Amex_vol_20d", "AMD_ret_1d", "DHR_ret_1d", "ITT_ITTInc_ret_5d", "ORCL_vol_20d"], "is_new": true}, {"model_id": "new_h1_STRESS_XGBoost_N12_t4", "algo": "XGBoost", "regime": "STRESS", "horizon": 1, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "MO_AltriaMG_ret_1d", "TM_Telephone_ret_1d", "3M_vol_20d", "BA_ret_1d", "EXC_Exelon_zscore_60d", "AXP_Amex_ret_20d", "CTAS_Cintas_vol_20d", "AXP_Amex_vol_20d", "HangSeng_HK_ret_5d", "Brent_Oil_FRED_ret_5d"], "is_new": true}, {"model_id": "new_h1_STRESS_XGBoost_N12_t5", "algo": "XGBoost", "regime": "STRESS", "horizon": 1, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EOG_EOGResources_ret_5d", "DHR_ret_1d", "US7Y_Rate_ret_20d", "PFE_ret_1d", "Core_PCE_zscore_60d", "DAX_Germany_zscore_60d", "INTC_ret_5d", "DIS_vol_20d", "NFCI_ret_5d", "heston_var_ev_h3"], "is_new": true}, {"model_id": "new_h1_STRESS_XGBoost_N12_t6", "algo": "XGBoost", "regime": "STRESS", "horizon": 1, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "spx_vol_5d", "SBUX_zscore_60d", "HD_ret_1d", "ITT_ITTInc_ret_5d", "SBUX_ret_5d", "EWM_Malaysia_ret_1d", "CMCSA_ret_1d", "FedFunds_zscore_60d", "HUM_Humana_ret_5d", "CI_Cigna_vol_20d"], "is_new": true}, {"model_id": "new_h1_STRESS_XGBoost_N12_t7", "algo": "XGBoost", "regime": "STRESS", "horizon": 1, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "Brent_Oil_FRED_ret_5d", "SCHW_Schwab_ret_5d", "BLK_BlackRock_zscore_60d", "EWJ_Japan_vol_20d", "AMD_ret_5d", "vix_mean_abs_ret_5d", "EWL_Switzerland_vol_20d", "PLD_Prologis_ret_5d", "MSTR_Bitcoin3_ret_5d", "EWA_Australia_ret_1d"], "is_new": true}, {"model_id": "new_h1_STRESS_XGBoost_N15_t0", "algo": "XGBoost", "regime": "STRESS", "horizon": 1, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "VRP_ma5", "TED_Spread_zscore_60d", "SO_SouthernCo_ret_5d", "INTC_ret_1d", "IBEX_Spain_ret_20d", "PCAR_PaccarInc_ret_5d", "spx_abs_ret_max_5d", "EFFR_ret_1d", "SLB_Schlumberger_ret_1d", "CPB_CampbellSoup_zscore_60d", "DE_Deere_ret_5d", "EWL_Switzerland_zscore_60d", "LUV_SouthwestAir_ret_5d"], "is_new": true}, {"model_id": "new_h1_STRESS_XGBoost_N15_t1", "algo": "XGBoost", "regime": "STRESS", "horizon": 1, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWA_Australia_ret_1d", "BTI_BritishAmerican_ret_5d", "HD_ret_1d", "NWL_Newell_ret_20d", "ENB_EnbridgeInc_ret_1d", "HangSeng_HK_vol_20d", "US5Y_Rate_ret_5d", "heston_var_ev_h7", "CPB_CampbellSoup_vol_20d", "DIS_vol_20d", "PG_ret_20d", "CPB_CampbellSoup_zscore_60d", "CTAS_Cintas_vol_20d"], "is_new": true}, {"model_id": "new_h1_STRESS_XGBoost_N15_t2", "algo": "XGBoost", "regime": "STRESS", "horizon": 1, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "GD_GeneralDynamics_zscore_60d", "AORD_AUS_zscore_60d", "QQQ_vol_20d", "spx_vol_5d", "TED_Spread_zscore_60d", "PAYX_Paychex_zscore_60d", "LMT_LockheedMartin_vol_20d", "FedFunds_zscore_60d", "Core_PCE_zscore_60d", "T_ret_1d", "CI_Cigna_vol_20d", "Nikkei_Japan_zscore_60d", "EWJ_Japan_vol_20d"], "is_new": true}, {"model_id": "new_h1_STRESS_XGBoost_N15_t3", "algo": "XGBoost", "regime": "STRESS", "horizon": 1, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "BA_ret_1d", "ORCL_zscore_60d", "GE_ret_1d", "MO_AltriaMG_ret_1d", "CPB_CampbellSoup_ret_5d", "AXP_Amex_vol_20d", "hmm_p_stress", "MSTR_Bitcoin3_ret_20d", "NWL_Newell_ret_20d", "EWA_Australia_zscore_60d", "HUM_Humana_ret_5d", "EWA_Australia_ret_1d", "Michigan_Sentiment_ret_20d"], "is_new": true}, {"model_id": "new_h1_STRESS_XGBoost_N15_t4", "algo": "XGBoost", "regime": "STRESS", "horizon": 1, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "CPB_CampbellSoup_zscore_60d", "AXP_Amex_vol_20d", "heston_var_ev_h7", "EWY_Korea_ret_20d", "CTAS_Cintas_vol_20d", "EWL_Switzerland_zscore_60d", "Core_CPI_zscore_60d", "BA_ret_1d", "ASX_Australia_vol_20d", "US5Y_Rate_ret_5d", "GD_GeneralDynamics_zscore_60d", "EWQ_France_zscore_60d", "BTI_BritishAmerican_ret_20d"], "is_new": true}, {"model_id": "new_h1_STRESS_XGBoost_N15_t5", "algo": "XGBoost", "regime": "STRESS", "horizon": 1, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "XLY_Disc_vol_20d", "Michigan_Sentiment_ret_20d", "MS_MorganStanley_ret_5d", "CPB_CampbellSoup_ret_5d", "EWL_Switzerland_zscore_60d", "GD_GeneralDynamics_zscore_60d", "SCHW_Schwab_ret_5d", "EWG_Germany_vol_20d", "EOG_EOGResources_ret_5d", "LLY_zscore_60d", "heston_var_ev_h3", "HD_ret_5d", "VVIX_ret_20d"], "is_new": true}, {"model_id": "new_h1_STRESS_XGBoost_N15_t6", "algo": "XGBoost", "regime": "STRESS", "horizon": 1, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "SJM_JM_Smucker_ret_5d", "XOM_ret_1d", "Retail_Sales_zscore_60d", "HangSeng_HK_vol_20d", "EWQ_France_zscore_60d", "EOG_EOGResources_ret_5d", "INTC_ret_1d", "BTI_BritishAmerican_ret_5d", "ENB_EnbridgeInc_ret_1d", "ES_Evergy_ret_1d", "IYR_US_REIT2_zscore_60d", "PCAR_PaccarInc_ret_5d", "DHR_vol_20d"], "is_new": true}, {"model_id": "new_h1_STRESS_XGBoost_N15_t7", "algo": "XGBoost", "regime": "STRESS", "horizon": 1, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "AMT_AmericanTower_ret_1d", "EQIX_Equinix_ret_5d", "FedFunds_zscore_60d", "Brent_Oil_FRED_ret_5d", "EOG_EOGResources_ret_5d", "AMD_ret_1d", "VVIX_ret_20d", "LMT_LockheedMartin_vol_20d", "QQQ_vol_20d", "US7Y_Rate_ret_20d", "3M_ret_5d", "SBUX_zscore_60d", "EWY_Korea_ret_20d"], "is_new": true}, {"model_id": "new_h1_STRESS_XGBoost_N20_t0", "algo": "XGBoost", "regime": "STRESS", "horizon": 1, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "US3M_Rate_zscore_60d", "HD_zscore_60d", "EWM_Malaysia_vol_20d", "TED_Spread_vol_20d", "SJM_JM_Smucker_ret_5d", "ENB_EnbridgeInc_ret_1d", "DIS_vol_20d", "CPB_CampbellSoup_ret_20d", "CPB_CampbellSoup_ret_5d", "HangSeng_HK_ret_5d", "IYR_US_REIT2_zscore_60d", "PAYX_Paychex_zscore_60d", "SBUX_zscore_60d", "LMT_LockheedMartin_ret_1d", "TXN_vol_20d", "EXC_Exelon_ret_1d", "spx_vol_5d", "SBUX_vol_20d"], "is_new": true}, {"model_id": "new_h1_STRESS_XGBoost_N20_t1", "algo": "XGBoost", "regime": "STRESS", "horizon": 1, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "LMT_LockheedMartin_vol_20d", "HangSeng_HK_ret_5d", "IYR_US_REIT2_zscore_60d", "SPY_zscore_60d", "spx_vol_5d", "VVIX_ret_20d", "SBUX_ret_5d", "CTAS_Cintas_vol_20d", "GE_ret_1d", "heston_var_ev_h3", "AMT_AmericanTower_ret_1d", "DIS_vol_20d", "NFCI_ret_5d", "EWL_Switzerland_zscore_60d", "spx_momentum_3d", "DHR_vol_20d", "LOW_Lowes_ret_20d", "SJM_JM_Smucker_ret_1d"], "is_new": true}, {"model_id": "new_h1_STRESS_XGBoost_N20_t2", "algo": "XGBoost", "regime": "STRESS", "horizon": 1, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWG_Germany_vol_20d", "CLX_Clorox_vol_20d", "EWA_Australia_zscore_60d", "MSTR_Bitcoin3_ret_1d", "heston_var_ev_h3", "CPB_CampbellSoup_zscore_60d", "GILD_Gilead_ret_20d", "XOM_ret_20d", "TXN_vol_20d", "PAYX_Paychex_zscore_60d", "TED_Spread_vol_20d", "EOG_EOGResources_vol_20d", "FedFunds_zscore_60d", "LMT_LockheedMartin_ret_1d", "IYM_BasicMaterials_ret_20d", "MS_MorganStanley_ret_5d", "VOD_Vodafone_zscore_60d", "NOC_Northrop_ret_20d"], "is_new": true}, {"model_id": "new_h1_STRESS_XGBoost_N20_t3", "algo": "XGBoost", "regime": "STRESS", "horizon": 1, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "US6M_Rate_ret_20d", "T10Y2Y_Spread_ret_5d", "JNJ_ret_1d", "XLK_Tech_zscore_60d", "DAX_Germany_vol_20d", "Nikkei_Japan_zscore_60d", "PCAR_PaccarInc_ret_5d", "US3M_Rate_zscore_60d", "LLY_zscore_60d", "HD_ret_5d", "CTAS_Cintas_vol_20d", "AMD_ret_1d", "SJM_JM_Smucker_ret_1d", "DIS_vol_20d", "PLD_Prologis_ret_5d", "DOW_Price_zscore_60d", "HD_zscore_60d", "XOM_ret_1d"], "is_new": true}, {"model_id": "new_h1_STRESS_XGBoost_N20_t4", "algo": "XGBoost", "regime": "STRESS", "horizon": 1, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "HD_ret_5d", "BTI_BritishAmerican_ret_20d", "HUM_Humana_ret_5d", "AMD_ret_5d", "PCAR_PaccarInc_ret_5d", "AVB_AvalonBay_zscore_60d", "EWL_Switzerland_vol_20d", "HD_ret_20d", "VRP_ma5", "DAX_Germany_vol_20d", "BA_ret_1d", "SBUX_zscore_60d", "HangSeng_HK_vol_20d", "3M_vol_20d", "Nikkei_Japan_zscore_60d", "LMT_LockheedMartin_vol_20d", "spx_vol_5d", "3M_ret_5d"], "is_new": true}, {"model_id": "new_h1_STRESS_XGBoost_N20_t5", "algo": "XGBoost", "regime": "STRESS", "horizon": 1, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "HD_ret_20d", "SJM_JM_Smucker_ret_5d", "vix_acceleration_1d", "US30Y_Rate_ret_20d", "EFFR_ret_1d", "CLX_Clorox_vol_20d", "GE_ret_1d", "DHR_vol_20d", "EWC_Canada_zscore_60d", "Nikkei_Japan_zscore_60d", "US7Y_Rate_ret_20d", "CCI_CrownCastle_vol_20d", "HD_zscore_60d", "EWL_Switzerland_zscore_60d", "ASX_Australia_ret_5d", "QQQ_vol_20d", "LMT_LockheedMartin_vol_20d", "EWY_Korea_ret_20d"], "is_new": true}, {"model_id": "new_h1_STRESS_XGBoost_N20_t6", "algo": "XGBoost", "regime": "STRESS", "horizon": 1, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "CPB_CampbellSoup_ret_5d", "EXC_Exelon_zscore_60d", "HangSeng_HK_ret_5d", "VVIX_ret_20d", "IBEX_Spain_ret_20d", "EOG_EOGResources_vol_20d", "US3M_Rate_vol_20d", "ORCL_vol_20d", "ITT_ITTInc_ret_5d", "US6M_Rate_ret_20d", "DHR_ret_1d", "EWG_Germany_ret_20d", "EWY_Korea_zscore_60d", "DE_Deere_vol_20d", "NOC_Northrop_ret_20d", "US1Y_Rate_ret_20d", "HangSeng_HK_vol_20d", "CI_Cigna_vol_20d"], "is_new": true}, {"model_id": "new_h1_STRESS_XGBoost_N20_t7", "algo": "XGBoost", "regime": "STRESS", "horizon": 1, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "ORCL_vol_20d", "CPB_CampbellSoup_ret_20d", "DHR_ret_1d", "TED_Spread_vol_20d", "PAYX_Paychex_ret_20d", "INTC_ret_1d", "SJM_JM_Smucker_ret_5d", "LOW_Lowes_ret_20d", "PLD_Prologis_ret_5d", "LMT_LockheedMartin_vol_20d", "Nikkei_Japan_vol_20d", "NVDA_vol_20d", "NFCI_ret_5d", "PG_ret_20d", "AORD_AUS_zscore_60d", "US30Y_Rate_ret_20d", "AMZN_ret_5d", "CLX_Clorox_vol_20d"], "is_new": true}, {"model_id": "new_h1_STRESS_XGBoost_N25_t0", "algo": "XGBoost", "regime": "STRESS", "horizon": 1, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "DAX_Germany_zscore_60d", "XLY_Disc_vol_20d", "TXN_vol_20d", "US3M_Rate_vol_20d", "HUM_Humana_ret_5d", "BA_ret_1d", "MS_MorganStanley_ret_5d", "vix_mean_abs_ret_5d", "Michigan_Sentiment_ret_20d", "3M_vol_20d", "PG_ret_20d", "SPY_zscore_60d", "PFE_ret_1d", "hmm_p_stress", "CI_Cigna_vol_20d", "BDX_Becton_Dickinson_ret_20d", "GE_ret_1d", "CPB_CampbellSoup_zscore_60d", "IYR_US_REIT2_zscore_60d", "VVIX_ret_20d", "CPB_CampbellSoup_ret_20d", "HD_zscore_60d", "spx_momentum_3d"], "is_new": true}, {"model_id": "new_h1_STRESS_XGBoost_N25_t1", "algo": "XGBoost", "regime": "STRESS", "horizon": 1, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "US7Y_Rate_ret_20d", "SLB_Schlumberger_ret_5d", "BA_ret_1d", "EWG_Germany_vol_20d", "SPY_zscore_60d", "DHR_vol_20d", "US30Y_Rate_ret_20d", "SCHW_Schwab_ret_5d", "DE_Deere_vol_20d", "EWM_Malaysia_ret_1d", "EFFR_ret_1d", "HangSeng_HK_ret_5d", "IBEX_Spain_ret_20d", "3M_vol_20d", "SBUX_vol_20d", "EWG_Germany_ret_20d", "Core_CPI_zscore_60d", "BTI_BritishAmerican_ret_5d", "3M_ret_5d", "gjr_condvar_h1", "Core_PCE_zscore_60d", "T10Y2Y_Spread_ret_5d", "CPB_CampbellSoup_ret_20d"], "is_new": true}, {"model_id": "new_h1_STRESS_XGBoost_N25_t2", "algo": "XGBoost", "regime": "STRESS", "horizon": 1, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "AMZN_ret_5d", "SLB_Schlumberger_ret_1d", "PAYX_Paychex_ret_20d", "MS_MorganStanley_ret_5d", "BDX_Becton_Dickinson_ret_20d", "DAX_Germany_vol_20d", "HangSeng_HK_ret_1d", "Michigan_Sentiment_ret_20d", "SBUX_ret_5d", "ES_Evergy_ret_1d", "EWQ_France_ret_20d", "TXN_vol_20d", "EXC_Exelon_zscore_60d", "XLK_Tech_zscore_60d", "ORCL_zscore_60d", "hmm_p_stress", "Nikkei_Japan_zscore_60d", "DIS_vol_20d", "AMT_AmericanTower_ret_1d", "heston_ev_h3", "DHR_vol_20d", "EWQ_France_zscore_60d", "EOG_EOGResources_ret_5d"], "is_new": true}, {"model_id": "new_h1_STRESS_XGBoost_N25_t3", "algo": "XGBoost", "regime": "STRESS", "horizon": 1, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "HD_zscore_60d", "vix_mean_abs_ret_5d", "CPB_CampbellSoup_ret_5d", "VOD_Vodafone_zscore_60d", "SCHW_Schwab_ret_5d", "MO_AltriaMG_ret_1d", "GILD_Gilead_ret_20d", "MS_MorganStanley_ret_5d", "MSTR_Bitcoin3_ret_5d", "CPB_CampbellSoup_zscore_60d", "PG_ret_20d", "NOC_Northrop_ret_20d", "NWL_Newell_ret_20d", "DIS_vol_20d", "AORD_AUS_zscore_60d", "HD_ret_20d", "PFE_ret_1d", "Core_PCE_zscore_60d", "US3M_Rate_vol_20d", "MS_MorganStanley_ret_1d", "AMD_ret_1d", "US6M_Rate_ret_20d", "HangSeng_HK_ret_5d"], "is_new": true}, {"model_id": "new_h1_STRESS_XGBoost_N25_t4", "algo": "XGBoost", "regime": "STRESS", "horizon": 1, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "DHR_ret_1d", "PLD_Prologis_ret_5d", "PFE_ret_1d", "GILD_Gilead_ret_20d", "AMD_ret_1d", "EFFR_vol_20d", "LOW_Lowes_ret_20d", "SJM_JM_Smucker_ret_5d", "heston_var_ev_h5", "EWH_HongKong_ret_5d", "LMT_LockheedMartin_ret_1d", "TED_Spread_vol_20d", "MS_MorganStanley_ret_1d", "SO_SouthernCo_ret_5d", "BTI_BritishAmerican_ret_20d", "EOG_EOGResources_vol_20d", "DOW_Price_zscore_60d", "LMT_LockheedMartin_vol_20d", "US3M_Rate_vol_20d", "Industrial_Production_zscore_60d", "spx_vol_5d", "vix_mean_abs_ret_5d", "HD_ret_1d"], "is_new": true}, {"model_id": "new_h1_STRESS_XGBoost_N25_t5", "algo": "XGBoost", "regime": "STRESS", "horizon": 1, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "heston_var_ev_h7", "SBUX_ret_5d", "MSTR_Bitcoin3_ret_1d", "PPL_PPL_ret_1d", "XLF_Fin_vol_20d", "EQR_Equity_ret_1d", "US5Y_Rate_ret_5d", "SLB_Schlumberger_ret_1d", "TED_Spread_vol_20d", "EXC_Exelon_zscore_60d", "DE_Deere_ret_5d", "ITT_ITTInc_ret_5d", "VOD_Vodafone_zscore_60d", "US6M_Rate_ret_20d", "LMT_LockheedMartin_ret_1d", "US7Y_Rate_ret_20d", "Michigan_Sentiment_ret_20d", "BTI_BritishAmerican_ret_5d", "AXP_Amex_ret_20d", "AORD_AUS_zscore_60d", "EFFR_vol_20d", "EWM_Malaysia_zscore_60d", "CPB_CampbellSoup_ret_20d"], "is_new": true}, {"model_id": "new_h1_STRESS_XGBoost_N25_t6", "algo": "XGBoost", "regime": "STRESS", "horizon": 1, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "ITT_ITTInc_ret_5d", "EWM_Malaysia_zscore_60d", "EWQ_France_ret_20d", "CPB_CampbellSoup_vol_20d", "EOG_EOGResources_vol_20d", "MSTR_Bitcoin3_ret_5d", "NVDA_vol_20d", "BTI_BritishAmerican_ret_20d", "BLK_BlackRock_zscore_60d", "XLK_Tech_zscore_60d", "EWY_Korea_zscore_60d", "Core_CPI_zscore_60d", "ASX_Australia_vol_20d", "Brent_Oil_FRED_ret_20d", "PAYX_Paychex_ret_20d", "MRK_Merck_zscore_60d", "XLF_Fin_vol_20d", "AORD_AUS_zscore_60d", "spx_abs_ret_max_5d", "vix_mean_abs_ret_5d", "CPB_CampbellSoup_zscore_60d", "PAYX_Paychex_zscore_60d", "MSTR_Bitcoin3_ret_1d"], "is_new": true}, {"model_id": "new_h1_STRESS_XGBoost_N25_t7", "algo": "XGBoost", "regime": "STRESS", "horizon": 1, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "SBUX_zscore_60d", "ORCL_zscore_60d", "HD_ret_5d", "EWM_Malaysia_ret_1d", "EWG_Germany_ret_20d", "EMR_Emerson_ret_20d", "ORCL_vol_20d", "TED_Spread_zscore_60d", "EWA_Australia_zscore_60d", "CLX_Clorox_vol_20d", "US7Y_Rate_ret_20d", "AMZN_ret_5d", "CPB_CampbellSoup_ret_20d", "EWG_Germany_vol_20d", "ASX_Australia_ret_5d", "DHR_ret_1d", "3M_vol_20d", "LOW_Lowes_ret_5d", "NEE_NextEra_ret_20d", "US3M_Rate_zscore_60d", "TXN_vol_20d", "Brent_Oil_FRED_ret_20d", "JNJ_ret_1d"], "is_new": true}, {"model_id": "new_h1_STRESS_XGBoost_N30_t0", "algo": "XGBoost", "regime": "STRESS", "horizon": 1, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "US3Y_Rate_ret_5d", "QQQ_vol_20d", "SJM_JM_Smucker_ret_5d", "LMT_LockheedMartin_vol_20d", "LOW_Lowes_ret_20d", "XLY_Disc_vol_20d", "US30Y_Rate_ret_20d", "DHR_ret_1d", "BTI_BritishAmerican_ret_5d", "ORCL_zscore_60d", "Nikkei_Japan_vol_20d", "XOM_ret_20d", "MSTR_Bitcoin3_ret_1d", "Brent_Oil_FRED_ret_20d", "US3M_Rate_zscore_60d", "CCI_CrownCastle_vol_20d", "DAX_Germany_zscore_60d", "SPY_zscore_60d", "HangSeng_HK_ret_1d", "CPB_CampbellSoup_ret_5d", "Core_PCE_zscore_60d", "SJM_JM_Smucker_ret_1d", "EWM_Malaysia_zscore_60d", "BA_ret_1d", "TM_Telephone_vol_20d", "IYM_BasicMaterials_ret_20d", "AMT_AmericanTower_ret_1d", "EWM_Malaysia_ret_1d"], "is_new": true}, {"model_id": "new_h1_STRESS_XGBoost_N30_t1", "algo": "XGBoost", "regime": "STRESS", "horizon": 1, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "BTI_BritishAmerican_ret_5d", "EFFR_vol_20d", "HD_ret_5d", "Brent_Oil_FRED_ret_5d", "TGT_Target_zscore_60d", "US5Y_Rate_ret_5d", "EWQ_France_zscore_60d", "SPY_zscore_60d", "XLK_Tech_zscore_60d", "SJM_JM_Smucker_ret_5d", "IBEX_Spain_ret_20d", "M_Macys_vol_20d", "XLY_Disc_vol_20d", "BDX_Becton_Dickinson_ret_20d", "TXN_vol_20d", "CTAS_Cintas_vol_20d", "US7Y_Rate_ret_20d", "PPL_PPL_ret_1d", "FedFunds_zscore_60d", "AORD_AUS_zscore_60d", "ASX_Australia_ret_5d", "Industrial_Production_zscore_60d", "NVDA_vol_20d", "EXC_Exelon_zscore_60d", "EWG_Germany_ret_20d", "IYR_US_REIT2_zscore_60d", "BLK_BlackRock_zscore_60d", "DHR_ret_1d"], "is_new": true}, {"model_id": "new_h1_STRESS_XGBoost_N30_t2", "algo": "XGBoost", "regime": "STRESS", "horizon": 1, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "SCHW_Schwab_ret_5d", "Brent_Oil_FRED_ret_20d", "US3M_Rate_vol_20d", "XLK_Tech_zscore_60d", "ASX_Australia_vol_20d", "LLY_zscore_60d", "vix_acceleration_1d", "NOC_Northrop_ret_20d", "WTI_Oil_FRED_zscore_60d", "spx_momentum_3d", "CCI_CrownCastle_vol_20d", "US1Y_Rate_ret_20d", "Retail_Sales_zscore_60d", "DE_Deere_ret_5d", "GE_ret_1d", "AVB_AvalonBay_zscore_60d", "CPB_CampbellSoup_vol_20d", "spx_abs_ret_max_5d", "SJM_JM_Smucker_ret_1d", "SLB_Schlumberger_ret_1d", "EWM_Malaysia_ret_1d", "LMT_LockheedMartin_ret_1d", "Nikkei_Japan_zscore_60d", "XOM_ret_1d", "HangSeng_HK_ret_5d", "EWM_Malaysia_vol_20d", "AMT_AmericanTower_ret_1d", "BDX_Becton_Dickinson_ret_20d"], "is_new": true}, {"model_id": "new_h1_STRESS_XGBoost_N30_t3", "algo": "XGBoost", "regime": "STRESS", "horizon": 1, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "US1Y_Rate_ret_20d", "SBUX_ret_5d", "BA_ret_1d", "MS_MorganStanley_ret_1d", "MSTR_Bitcoin3_ret_1d", "DAX_Germany_zscore_60d", "Nikkei_Japan_zscore_60d", "Nikkei_Japan_vol_20d", "T_ret_1d", "DHR_ret_1d", "Core_CPI_zscore_60d", "ORCL_zscore_60d", "Core_PCE_zscore_60d", "XLV_Health_zscore_60d", "MO_AltriaMG_ret_1d", "LMT_LockheedMartin_vol_20d", "EWA_Australia_zscore_60d", "JNJ_ret_1d", "ORCL_vol_20d", "BDX_Becton_Dickinson_ret_20d", "EWQ_France_ret_20d", "EWM_Malaysia_zscore_60d", "CMCSA_ret_1d", "AMD_ret_1d", "EWM_Malaysia_vol_20d", "NFCI_ret_5d", "TM_Telephone_ret_1d", "AXP_Amex_ret_20d"], "is_new": true}, {"model_id": "new_h1_STRESS_XGBoost_N30_t4", "algo": "XGBoost", "regime": "STRESS", "horizon": 1, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "BDX_Becton_Dickinson_ret_20d", "DAX_Germany_zscore_60d", "ORCL_zscore_60d", "EQR_Equity_ret_1d", "HD_ret_20d", "AXP_Amex_ret_20d", "INTC_ret_1d", "LMT_LockheedMartin_ret_1d", "XLB_Materials_zscore_60d", "EWS_Singapore_ret_5d", "HD_zscore_60d", "DE_Deere_ret_5d", "XLF_Fin_vol_20d", "MS_MorganStanley_zscore_60d", "Retail_Sales_zscore_60d", "DAX_Germany_vol_20d", "Brent_Oil_FRED_ret_5d", "XOM_ret_1d", "EWY_Korea_zscore_60d", "TXN_vol_20d", "EWG_Germany_vol_20d", "SCHW_Schwab_ret_5d", "LOW_Lowes_ret_20d", "EWL_Switzerland_zscore_60d", "PAYX_Paychex_ret_20d", "HangSeng_HK_ret_5d", "LUV_SouthwestAir_ret_5d", "US1Y_Rate_ret_20d"], "is_new": true}, {"model_id": "new_h1_STRESS_XGBoost_N30_t5", "algo": "XGBoost", "regime": "STRESS", "horizon": 1, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "TM_Telephone_vol_20d", "SJM_JM_Smucker_ret_5d", "CPB_CampbellSoup_vol_20d", "BA_ret_1d", "LLY_zscore_60d", "NFCI_ret_5d", "XOM_ret_1d", "SBUX_vol_20d", "HD_ret_1d", "CCI_CrownCastle_vol_20d", "EWG_Germany_vol_20d", "DHR_ret_1d", "T10Y2Y_Spread_ret_5d", "EWM_Malaysia_zscore_60d", "HangSeng_HK_vol_20d", "US1Y_Rate_ret_20d", "EWH_HongKong_ret_5d", "DAX_Germany_vol_20d", "MRK_Merck_zscore_60d", "DHR_vol_20d", "ASX_Australia_vol_20d", "EWQ_France_ret_20d", "ORCL_zscore_60d", "PCAR_PaccarInc_ret_5d", "Core_CPI_zscore_60d", "AMT_AmericanTower_ret_1d", "SJM_JM_Smucker_ret_1d", "ASX_Australia_ret_5d"], "is_new": true}, {"model_id": "new_h1_STRESS_XGBoost_N30_t6", "algo": "XGBoost", "regime": "STRESS", "horizon": 1, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "CI_Cigna_vol_20d", "BLK_BlackRock_zscore_60d", "HD_ret_1d", "XLK_Tech_zscore_60d", "MSTR_Bitcoin3_ret_5d", "SCHW_Schwab_ret_5d", "AMGN_Amgen_ret_1d", "MSTR_Bitcoin3_ret_20d", "spx_abs_ret_max_5d", "PAYX_Paychex_ret_20d", "AXP_Amex_ret_20d", "EWM_Malaysia_ret_1d", "EWG_Germany_vol_20d", "heston_var_ev_h5", "ES_Evergy_ret_1d", "Nikkei_Japan_zscore_60d", "TGT_Target_zscore_60d", "BA_ret_1d", "HangSeng_HK_vol_20d", "ASX_Australia_ret_5d", "TM_Telephone_vol_20d", "LMT_LockheedMartin_ret_1d", "AMZN_ret_5d", "Industrial_Production_zscore_60d", "FedFunds_zscore_60d", "EWL_Switzerland_vol_20d", "AORD_AUS_zscore_60d", "JNJ_ret_1d"], "is_new": true}, {"model_id": "new_h1_STRESS_XGBoost_N30_t7", "algo": "XGBoost", "regime": "STRESS", "horizon": 1, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "CPB_CampbellSoup_zscore_60d", "US1Y_Rate_ret_20d", "MS_MorganStanley_ret_1d", "HD_ret_5d", "TGT_Target_zscore_60d", "PG_ret_20d", "NWL_Newell_ret_20d", "US3Y_Rate_ret_5d", "US3M_Rate_zscore_60d", "SPY_zscore_60d", "EXC_Exelon_zscore_60d", "EMR_Emerson_ret_20d", "MSTR_Bitcoin3_ret_20d", "PLD_Prologis_ret_5d", "EWQ_France_ret_20d", "DE_Deere_ret_5d", "AMD_ret_5d", "HD_ret_1d", "GILD_Gilead_ret_20d", "CI_Cigna_vol_20d", "EWY_Korea_ret_20d", "TM_Telephone_vol_20d", "HUM_Humana_ret_5d", "Industrial_Production_zscore_60d", "heston_var_ev_h5", "MS_MorganStanley_zscore_60d", "ASX_Australia_vol_20d", "WTI_Oil_FRED_zscore_60d"], "is_new": true}, {"model_id": "new_h1_STRESS_LightGBM_N5_t0", "algo": "LightGBM", "regime": "STRESS", "horizon": 1, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "DAX_Germany_vol_20d", "HangSeng_HK_vol_20d", "US3M_Rate_zscore_60d"], "is_new": true}, {"model_id": "new_h1_STRESS_LightGBM_N5_t1", "algo": "LightGBM", "regime": "STRESS", "horizon": 1, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "PAYX_Paychex_ret_20d", "PG_ret_20d", "AMGN_Amgen_ret_1d"], "is_new": true}, {"model_id": "new_h1_STRESS_LightGBM_N5_t2", "algo": "LightGBM", "regime": "STRESS", "horizon": 1, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "LOW_Lowes_ret_20d", "Retail_Sales_zscore_60d", "heston_var_ev_h3"], "is_new": true}, {"model_id": "new_h1_STRESS_LightGBM_N5_t3", "algo": "LightGBM", "regime": "STRESS", "horizon": 1, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "US6M_Rate_ret_20d", "CCI_CrownCastle_vol_20d", "GILD_Gilead_ret_20d"], "is_new": true}, {"model_id": "new_h1_STRESS_LightGBM_N5_t4", "algo": "LightGBM", "regime": "STRESS", "horizon": 1, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EQR_Equity_ret_1d", "EXC_Exelon_zscore_60d", "GILD_Gilead_ret_20d"], "is_new": true}, {"model_id": "new_h1_STRESS_LightGBM_N5_t5", "algo": "LightGBM", "regime": "STRESS", "horizon": 1, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "gjr_condvar_h1", "EOG_EOGResources_vol_20d", "DIS_vol_20d"], "is_new": true}, {"model_id": "new_h1_STRESS_LightGBM_N5_t6", "algo": "LightGBM", "regime": "STRESS", "horizon": 1, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "spx_abs_ret_max_5d", "ASX_Australia_vol_20d", "ES_Evergy_ret_1d"], "is_new": true}, {"model_id": "new_h1_STRESS_LightGBM_N5_t7", "algo": "LightGBM", "regime": "STRESS", "horizon": 1, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "US1Y_Rate_ret_20d", "EWM_Malaysia_vol_20d", "BLK_BlackRock_zscore_60d"], "is_new": true}, {"model_id": "new_h1_STRESS_LightGBM_N8_t0", "algo": "LightGBM", "regime": "STRESS", "horizon": 1, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "LMT_LockheedMartin_vol_20d", "TM_Telephone_vol_20d", "AXP_Amex_ret_20d", "WTI_Oil_FRED_zscore_60d", "EWQ_France_ret_20d", "LMT_LockheedMartin_ret_1d"], "is_new": true}, {"model_id": "new_h1_STRESS_LightGBM_N8_t1", "algo": "LightGBM", "regime": "STRESS", "horizon": 1, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "SLB_Schlumberger_ret_5d", "Brent_Oil_FRED_ret_5d", "EWQ_France_ret_20d", "EWG_Germany_vol_20d", "IWM_SmallCap_vol_20d", "US3Y_Rate_ret_5d"], "is_new": true}, {"model_id": "new_h1_STRESS_LightGBM_N8_t2", "algo": "LightGBM", "regime": "STRESS", "horizon": 1, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "HD_ret_20d", "DHR_vol_20d", "AMZN_ret_5d", "IWM_SmallCap_vol_20d", "XLF_Fin_vol_20d", "LOW_Lowes_ret_5d"], "is_new": true}, {"model_id": "new_h1_STRESS_LightGBM_N8_t3", "algo": "LightGBM", "regime": "STRESS", "horizon": 1, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "HUM_Humana_ret_5d", "XOM_ret_1d", "AORD_AUS_zscore_60d", "EMR_Emerson_ret_20d", "MO_AltriaMG_ret_1d", "AXP_Amex_ret_20d"], "is_new": true}, {"model_id": "new_h1_STRESS_LightGBM_N8_t4", "algo": "LightGBM", "regime": "STRESS", "horizon": 1, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "XLF_Fin_vol_20d", "EWJ_Japan_vol_20d", "INTC_ret_1d", "SLB_Schlumberger_ret_5d", "US6M_Rate_ret_20d", "LLY_zscore_60d"], "is_new": true}, {"model_id": "new_h1_STRESS_LightGBM_N8_t5", "algo": "LightGBM", "regime": "STRESS", "horizon": 1, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "MSTR_Bitcoin3_ret_20d", "hmm_p_stress", "CCI_CrownCastle_vol_20d", "Brent_Oil_FRED_ret_5d", "EWM_Malaysia_ret_1d", "US3M_Rate_vol_20d"], "is_new": true}, {"model_id": "new_h1_STRESS_LightGBM_N8_t6", "algo": "LightGBM", "regime": "STRESS", "horizon": 1, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "XLF_Fin_vol_20d", "PPL_PPL_ret_1d", "spx_momentum_3d", "TED_Spread_vol_20d", "T10Y2Y_Spread_ret_5d", "ORCL_zscore_60d"], "is_new": true}, {"model_id": "new_h1_STRESS_LightGBM_N8_t7", "algo": "LightGBM", "regime": "STRESS", "horizon": 1, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWS_Singapore_ret_5d", "IYM_BasicMaterials_ret_20d", "LOW_Lowes_ret_5d", "EWM_Malaysia_zscore_60d", "PCAR_PaccarInc_ret_5d", "INTC_ret_1d"], "is_new": true}, {"model_id": "new_h1_STRESS_LightGBM_N10_t0", "algo": "LightGBM", "regime": "STRESS", "horizon": 1, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "NEE_NextEra_ret_20d", "LOW_Lowes_ret_5d", "CCI_CrownCastle_vol_20d", "LMT_LockheedMartin_vol_20d", "US30Y_Rate_ret_20d", "SLB_Schlumberger_ret_1d", "Brent_Oil_FRED_ret_5d", "spx_abs_ret_max_5d"], "is_new": true}, {"model_id": "new_h1_STRESS_LightGBM_N10_t1", "algo": "LightGBM", "regime": "STRESS", "horizon": 1, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWM_Malaysia_vol_20d", "US3M_Rate_zscore_60d", "DAX_Germany_zscore_60d", "TM_Telephone_ret_1d", "EWS_Singapore_ret_5d", "NEE_NextEra_ret_20d", "US7Y_Rate_ret_20d", "vix_acceleration_1d"], "is_new": true}, {"model_id": "new_h1_STRESS_LightGBM_N10_t2", "algo": "LightGBM", "regime": "STRESS", "horizon": 1, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "INTC_ret_1d", "Core_PCE_zscore_60d", "CCI_CrownCastle_vol_20d", "NEE_NextEra_ret_20d", "NVDA_vol_20d", "SLB_Schlumberger_ret_1d", "DIS_vol_20d", "US5Y_Rate_ret_5d"], "is_new": true}, {"model_id": "new_h1_STRESS_LightGBM_N10_t3", "algo": "LightGBM", "regime": "STRESS", "horizon": 1, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "AMD_ret_5d", "ENB_EnbridgeInc_ret_1d", "Core_PCE_zscore_60d", "NVDA_vol_20d", "FedFunds_zscore_60d", "US1Y_Rate_ret_20d", "PPL_PPL_ret_1d", "CPB_CampbellSoup_ret_5d"], "is_new": true}, {"model_id": "new_h1_STRESS_LightGBM_N10_t4", "algo": "LightGBM", "regime": "STRESS", "horizon": 1, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "ENB_EnbridgeInc_ret_1d", "Brent_Oil_FRED_ret_20d", "DE_Deere_ret_5d", "SO_SouthernCo_ret_5d", "TED_Spread_zscore_60d", "MS_MorganStanley_ret_1d", "NWL_Newell_ret_20d", "LMT_LockheedMartin_vol_20d"], "is_new": true}, {"model_id": "new_h1_STRESS_LightGBM_N10_t5", "algo": "LightGBM", "regime": "STRESS", "horizon": 1, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "Core_PCE_zscore_60d", "Industrial_Production_zscore_60d", "VRP_ma5", "XLF_Fin_vol_20d", "XLV_Health_zscore_60d", "CPB_CampbellSoup_zscore_60d", "AXP_Amex_ret_20d", "Brent_Oil_FRED_ret_20d"], "is_new": true}, {"model_id": "new_h1_STRESS_LightGBM_N10_t6", "algo": "LightGBM", "regime": "STRESS", "horizon": 1, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "gjr_condvar_h1", "PAYX_Paychex_zscore_60d", "ORCL_vol_20d", "HD_zscore_60d", "Retail_Sales_zscore_60d", "DE_Deere_vol_20d", "EQR_Equity_ret_1d", "VOD_Vodafone_zscore_60d"], "is_new": true}, {"model_id": "new_h1_STRESS_LightGBM_N10_t7", "algo": "LightGBM", "regime": "STRESS", "horizon": 1, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "VRP_ma5", "MO_AltriaMG_ret_1d", "hmm_p_stress", "PCAR_PaccarInc_ret_5d", "MSTR_Bitcoin3_ret_20d", "ITT_ITTInc_ret_5d", "ORCL_zscore_60d", "EQIX_Equinix_ret_5d"], "is_new": true}, {"model_id": "new_h1_STRESS_LightGBM_N12_t0", "algo": "LightGBM", "regime": "STRESS", "horizon": 1, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "HangSeng_HK_ret_1d", "SCHW_Schwab_ret_5d", "JNJ_ret_1d", "AXP_Amex_vol_20d", "US3M_Rate_vol_20d", "SPY_zscore_60d", "TED_Spread_vol_20d", "US6M_Rate_ret_20d", "SLB_Schlumberger_ret_5d", "Nikkei_Japan_vol_20d"], "is_new": true}, {"model_id": "new_h1_STRESS_LightGBM_N12_t1", "algo": "LightGBM", "regime": "STRESS", "horizon": 1, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EQIX_Equinix_ret_5d", "DHR_vol_20d", "AVB_AvalonBay_zscore_60d", "EWH_HongKong_ret_5d", "Nikkei_Japan_zscore_60d", "SPY_zscore_60d", "HD_ret_1d", "US5Y_Rate_ret_5d", "XLV_Health_zscore_60d", "INTC_ret_1d"], "is_new": true}, {"model_id": "new_h1_STRESS_LightGBM_N12_t2", "algo": "LightGBM", "regime": "STRESS", "horizon": 1, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "Core_PCE_zscore_60d", "HD_ret_5d", "GILD_Gilead_ret_20d", "SBUX_vol_20d", "ORCL_zscore_60d", "EWM_Malaysia_ret_1d", "EOG_EOGResources_ret_5d", "EXC_Exelon_zscore_60d", "XLF_Fin_vol_20d", "CPB_CampbellSoup_zscore_60d"], "is_new": true}, {"model_id": "new_h1_STRESS_LightGBM_N12_t3", "algo": "LightGBM", "regime": "STRESS", "horizon": 1, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWG_Germany_vol_20d", "US1Y_Rate_ret_20d", "SLB_Schlumberger_ret_5d", "EOG_EOGResources_ret_5d", "AORD_AUS_zscore_60d", "WTI_Oil_FRED_zscore_60d", "AVB_AvalonBay_zscore_60d", "TM_Telephone_vol_20d", "TM_Telephone_ret_1d", "EWA_Australia_zscore_60d"], "is_new": true}, {"model_id": "new_h1_STRESS_LightGBM_N12_t4", "algo": "LightGBM", "regime": "STRESS", "horizon": 1, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWG_Germany_vol_20d", "HD_ret_20d", "US3Y_Rate_ret_5d", "Nikkei_Japan_vol_20d", "PAYX_Paychex_ret_20d", "T10Y2Y_Spread_ret_5d", "DAX_Germany_zscore_60d", "PCAR_PaccarInc_ret_5d", "HUM_Humana_ret_5d", "CTAS_Cintas_vol_20d"], "is_new": true}, {"model_id": "new_h1_STRESS_LightGBM_N12_t5", "algo": "LightGBM", "regime": "STRESS", "horizon": 1, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "SCHW_Schwab_ret_5d", "FedFunds_zscore_60d", "CLX_Clorox_vol_20d", "DHR_ret_1d", "HD_ret_1d", "vix_mean_abs_ret_5d", "Core_CPI_zscore_60d", "LMT_LockheedMartin_vol_20d", "CCI_CrownCastle_vol_20d", "EWL_Switzerland_zscore_60d"], "is_new": true}, {"model_id": "new_h1_STRESS_LightGBM_N12_t6", "algo": "LightGBM", "regime": "STRESS", "horizon": 1, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "SLB_Schlumberger_ret_1d", "IYR_US_REIT2_zscore_60d", "GILD_Gilead_ret_20d", "PCAR_PaccarInc_ret_5d", "gjr_condvar_h1", "SO_SouthernCo_ret_5d", "Brent_Oil_FRED_ret_20d", "INTC_ret_1d", "ASX_Australia_vol_20d", "Nikkei_Japan_zscore_60d"], "is_new": true}, {"model_id": "new_h1_STRESS_LightGBM_N12_t7", "algo": "LightGBM", "regime": "STRESS", "horizon": 1, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "heston_ev_h3", "XOM_ret_20d", "AMD_ret_1d", "EWS_Singapore_ret_5d", "DHR_ret_1d", "VOD_Vodafone_zscore_60d", "MSTR_Bitcoin3_ret_1d", "EWL_Switzerland_zscore_60d", "EWC_Canada_zscore_60d", "FedFunds_zscore_60d"], "is_new": true}, {"model_id": "new_h1_STRESS_LightGBM_N15_t0", "algo": "LightGBM", "regime": "STRESS", "horizon": 1, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "SJM_JM_Smucker_ret_1d", "HD_ret_20d", "DE_Deere_ret_5d", "EWC_Canada_zscore_60d", "heston_ev_h3", "CLX_Clorox_vol_20d", "SPY_zscore_60d", "EWJ_Japan_vol_20d", "3M_ret_5d", "ORCL_vol_20d", "BTI_BritishAmerican_ret_5d", "SBUX_zscore_60d", "US1Y_Rate_ret_5d"], "is_new": true}, {"model_id": "new_h1_STRESS_LightGBM_N15_t1", "algo": "LightGBM", "regime": "STRESS", "horizon": 1, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "NVDA_vol_20d", "FedFunds_zscore_60d", "XLV_Health_zscore_60d", "MSTR_Bitcoin3_ret_20d", "gjr_condvar_h1", "US7Y_Rate_ret_20d", "EWQ_France_ret_20d", "heston_var_ev_h3", "LMT_LockheedMartin_ret_1d", "SBUX_ret_5d", "TED_Spread_zscore_60d", "spx_momentum_3d", "EQR_Equity_ret_1d"], "is_new": true}, {"model_id": "new_h1_STRESS_LightGBM_N15_t2", "algo": "LightGBM", "regime": "STRESS", "horizon": 1, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "SBUX_zscore_60d", "gjr_condvar_h1", "MS_MorganStanley_ret_5d", "WTI_Oil_FRED_zscore_60d", "LOW_Lowes_ret_20d", "TXN_vol_20d", "SBUX_ret_5d", "HUM_Humana_ret_5d", "CPB_CampbellSoup_ret_20d", "EWL_Switzerland_zscore_60d", "CTAS_Cintas_vol_20d", "HD_ret_5d", "NVDA_vol_20d"], "is_new": true}, {"model_id": "new_h1_STRESS_LightGBM_N15_t3", "algo": "LightGBM", "regime": "STRESS", "horizon": 1, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "Core_CPI_zscore_60d", "EQIX_Equinix_ret_5d", "PG_ret_20d", "INTC_ret_5d", "spx_vol_5d", "LLY_zscore_60d", "MS_MorganStanley_zscore_60d", "DAX_Germany_zscore_60d", "SBUX_ret_5d", "SBUX_zscore_60d", "WTI_Oil_FRED_zscore_60d", "NFCI_ret_5d", "EWQ_France_ret_20d"], "is_new": true}, {"model_id": "new_h1_STRESS_LightGBM_N15_t4", "algo": "LightGBM", "regime": "STRESS", "horizon": 1, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "NEE_NextEra_ret_20d", "BLK_BlackRock_zscore_60d", "US30Y_Rate_ret_20d", "ITT_ITTInc_ret_5d", "EXC_Exelon_zscore_60d", "INTC_ret_5d", "NFCI_ret_5d", "MS_MorganStanley_ret_5d", "PAYX_Paychex_ret_20d", "NWL_Newell_ret_20d", "M_Macys_vol_20d", "BA_ret_1d", "GD_GeneralDynamics_zscore_60d"], "is_new": true}, {"model_id": "new_h1_STRESS_LightGBM_N15_t5", "algo": "LightGBM", "regime": "STRESS", "horizon": 1, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "HangSeng_HK_vol_20d", "PAYX_Paychex_vol_20d", "NEE_NextEra_ret_20d", "EWL_Switzerland_zscore_60d", "US3Y_Rate_ret_5d", "ENB_EnbridgeInc_ret_1d", "TM_Telephone_ret_1d", "EWG_Germany_vol_20d", "XOM_ret_20d", "CTAS_Cintas_vol_20d", "XLV_Health_zscore_60d", "XLK_Tech_zscore_60d", "IYM_BasicMaterials_ret_20d"], "is_new": true}, {"model_id": "new_h1_STRESS_LightGBM_N15_t6", "algo": "LightGBM", "regime": "STRESS", "horizon": 1, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "CMCSA_ret_1d", "IYR_US_REIT2_zscore_60d", "LMT_LockheedMartin_vol_20d", "US1Y_Rate_ret_20d", "BTI_BritishAmerican_ret_20d", "spx_abs_ret_max_5d", "AXP_Amex_vol_20d", "SPY_zscore_60d", "EWH_HongKong_ret_5d", "LOW_Lowes_ret_5d", "TM_Telephone_ret_1d", "EWC_Canada_zscore_60d", "EFFR_vol_20d"], "is_new": true}, {"model_id": "new_h1_STRESS_LightGBM_N15_t7", "algo": "LightGBM", "regime": "STRESS", "horizon": 1, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EMR_Emerson_ret_20d", "MRK_Merck_zscore_60d", "IYM_BasicMaterials_ret_20d", "NFCI_ret_5d", "EWL_Switzerland_vol_20d", "US1Y_Rate_ret_20d", "EWY_Korea_ret_20d", "AMD_ret_5d", "ASX_Australia_vol_20d", "AMT_AmericanTower_ret_1d", "TED_Spread_vol_20d", "CPB_CampbellSoup_vol_20d", "SJM_JM_Smucker_ret_5d"], "is_new": true}, {"model_id": "new_h1_STRESS_LightGBM_N20_t0", "algo": "LightGBM", "regime": "STRESS", "horizon": 1, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "HD_ret_20d", "HangSeng_HK_ret_5d", "AMZN_ret_5d", "EWM_Malaysia_vol_20d", "gjr_condvar_h1", "EWY_Korea_ret_20d", "ASX_Australia_ret_5d", "GE_ret_1d", "DAX_Germany_vol_20d", "MSTR_Bitcoin3_ret_1d", "spx_abs_ret_max_5d", "M_Macys_vol_20d", "DE_Deere_ret_5d", "CI_Cigna_vol_20d", "Brent_Oil_FRED_ret_5d", "Brent_Oil_FRED_ret_20d", "SLB_Schlumberger_ret_1d", "NOC_Northrop_ret_20d"], "is_new": true}, {"model_id": "new_h1_STRESS_LightGBM_N20_t1", "algo": "LightGBM", "regime": "STRESS", "horizon": 1, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "VOD_Vodafone_zscore_60d", "MS_MorganStanley_ret_5d", "EWA_Australia_zscore_60d", "XLF_Fin_vol_20d", "IBEX_Spain_ret_20d", "ITT_ITTInc_ret_5d", "NWL_Newell_ret_20d", "SO_SouthernCo_ret_5d", "AMT_AmericanTower_ret_1d", "HangSeng_HK_ret_5d", "GILD_Gilead_ret_20d", "PFE_ret_1d", "EWG_Germany_ret_20d", "CPB_CampbellSoup_ret_5d", "FedFunds_zscore_60d", "spx_abs_ret_max_5d", "US5Y_Rate_ret_5d", "PPL_PPL_ret_1d"], "is_new": true}, {"model_id": "new_h1_STRESS_LightGBM_N20_t2", "algo": "LightGBM", "regime": "STRESS", "horizon": 1, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWL_Switzerland_zscore_60d", "NEE_NextEra_ret_20d", "US30Y_Rate_ret_20d", "HD_ret_1d", "SBUX_ret_5d", "HD_ret_20d", "TGT_Target_zscore_60d", "SLB_Schlumberger_ret_1d", "EWS_Singapore_ret_5d", "MSTR_Bitcoin3_ret_20d", "AMGN_Amgen_ret_1d", "SJM_JM_Smucker_ret_1d", "MS_MorganStanley_ret_5d", "MRK_Merck_zscore_60d", "TXN_vol_20d", "CPB_CampbellSoup_zscore_60d", "PAYX_Paychex_zscore_60d", "EWM_Malaysia_ret_1d"], "is_new": true}, {"model_id": "new_h1_STRESS_LightGBM_N20_t3", "algo": "LightGBM", "regime": "STRESS", "horizon": 1, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "PAYX_Paychex_vol_20d", "MO_AltriaMG_ret_1d", "Retail_Sales_zscore_60d", "US3M_Rate_vol_20d", "CPB_CampbellSoup_zscore_60d", "heston_var_ev_h5", "IBEX_Spain_ret_20d", "IYR_US_REIT2_zscore_60d", "DHR_vol_20d", "gjr_condvar_h1", "CI_Cigna_vol_20d", "CPB_CampbellSoup_ret_20d", "US30Y_Rate_ret_20d", "EOG_EOGResources_ret_5d", "EWH_HongKong_ret_5d", "SBUX_zscore_60d", "VRP_ma5", "EWY_Korea_ret_20d"], "is_new": true}, {"model_id": "new_h1_STRESS_LightGBM_N20_t4", "algo": "LightGBM", "regime": "STRESS", "horizon": 1, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "T10Y2Y_Spread_ret_5d", "US3M_Rate_zscore_60d", "EWM_Malaysia_zscore_60d", "BLK_BlackRock_zscore_60d", "SCHW_Schwab_ret_5d", "EWY_Korea_zscore_60d", "NFCI_ret_5d", "IBEX_Spain_ret_20d", "Industrial_Production_zscore_60d", "DOW_Price_zscore_60d", "ASX_Australia_vol_20d", "ITT_ITTInc_ret_5d", "US3M_Rate_vol_20d", "WTI_Oil_FRED_zscore_60d", "NEE_NextEra_ret_20d", "3M_vol_20d", "GD_GeneralDynamics_zscore_60d", "EWH_HongKong_ret_5d"], "is_new": true}, {"model_id": "new_h1_STRESS_LightGBM_N20_t5", "algo": "LightGBM", "regime": "STRESS", "horizon": 1, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "hmm_p_stress", "EWQ_France_ret_20d", "ITT_ITTInc_ret_5d", "PG_ret_20d", "SBUX_ret_5d", "SO_SouthernCo_ret_5d", "US1Y_Rate_ret_5d", "CCI_CrownCastle_vol_20d", "CPB_CampbellSoup_vol_20d", "NWL_Newell_ret_20d", "Brent_Oil_FRED_ret_5d", "MSTR_Bitcoin3_ret_1d", "DAX_Germany_zscore_60d", "AMD_ret_5d", "EMR_Emerson_ret_20d", "T10Y2Y_Spread_ret_5d", "DAX_Germany_vol_20d", "ENB_EnbridgeInc_ret_1d"], "is_new": true}, {"model_id": "new_h1_STRESS_LightGBM_N20_t6", "algo": "LightGBM", "regime": "STRESS", "horizon": 1, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "NWL_Newell_ret_20d", "SCHW_Schwab_ret_5d", "EOG_EOGResources_ret_5d", "XOM_ret_20d", "TED_Spread_vol_20d", "SBUX_vol_20d", "DHR_ret_1d", "T_ret_1d", "NVDA_vol_20d", "M_Macys_vol_20d", "AMD_ret_1d", "HUM_Humana_ret_5d", "Core_PCE_zscore_60d", "ORCL_vol_20d", "GE_ret_1d", "T10Y2Y_Spread_ret_5d", "ITT_ITTInc_ret_5d", "XLV_Health_zscore_60d"], "is_new": true}, {"model_id": "new_h1_STRESS_LightGBM_N20_t7", "algo": "LightGBM", "regime": "STRESS", "horizon": 1, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "T10Y2Y_Spread_ret_5d", "EWG_Germany_vol_20d", "XLV_Health_zscore_60d", "MRK_Merck_zscore_60d", "SPY_zscore_60d", "XLB_Materials_zscore_60d", "EWY_Korea_zscore_60d", "PPL_PPL_ret_1d", "PG_ret_20d", "EWH_HongKong_ret_5d", "IYM_BasicMaterials_ret_20d", "DE_Deere_vol_20d", "ENB_EnbridgeInc_ret_1d", "TED_Spread_vol_20d", "QQQ_vol_20d", "AMD_ret_1d", "INTC_ret_5d", "NOC_Northrop_ret_20d"], "is_new": true}, {"model_id": "new_h1_STRESS_LightGBM_N25_t0", "algo": "LightGBM", "regime": "STRESS", "horizon": 1, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "3M_vol_20d", "IBEX_Spain_ret_20d", "EWA_Australia_ret_1d", "EWG_Germany_ret_20d", "CPB_CampbellSoup_ret_20d", "TM_Telephone_ret_1d", "PFE_ret_1d", "IYM_BasicMaterials_ret_20d", "spx_abs_ret_max_5d", "BLK_BlackRock_zscore_60d", "XOM_ret_1d", "PLD_Prologis_ret_5d", "VVIX_ret_20d", "EFFR_vol_20d", "BTI_BritishAmerican_ret_5d", "LMT_LockheedMartin_ret_1d", "US30Y_Rate_ret_20d", "EWM_Malaysia_ret_1d", "LLY_zscore_60d", "AMD_ret_5d", "EWA_Australia_zscore_60d", "GILD_Gilead_ret_20d", "HD_ret_20d"], "is_new": true}, {"model_id": "new_h1_STRESS_LightGBM_N25_t1", "algo": "LightGBM", "regime": "STRESS", "horizon": 1, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EXC_Exelon_ret_1d", "US30Y_Rate_ret_20d", "TM_Telephone_vol_20d", "heston_var_ev_h5", "EWG_Germany_ret_20d", "Brent_Oil_FRED_ret_5d", "VRP_ma5", "EFFR_ret_1d", "CPB_CampbellSoup_vol_20d", "EWL_Switzerland_vol_20d", "HD_ret_1d", "EQIX_Equinix_ret_5d", "FedFunds_zscore_60d", "US3M_Rate_vol_20d", "heston_var_ev_h3", "DAX_Germany_zscore_60d", "T10Y2Y_Spread_ret_5d", "VOD_Vodafone_zscore_60d", "MSTR_Bitcoin3_ret_5d", "IYM_BasicMaterials_ret_20d", "US5Y_Rate_ret_5d", "GILD_Gilead_ret_20d", "QQQ_vol_20d"], "is_new": true}, {"model_id": "new_h1_STRESS_LightGBM_N25_t2", "algo": "LightGBM", "regime": "STRESS", "horizon": 1, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWL_Switzerland_vol_20d", "CMCSA_ret_1d", "XLV_Health_zscore_60d", "DIS_vol_20d", "DHR_vol_20d", "ENB_EnbridgeInc_ret_1d", "QQQ_vol_20d", "3M_ret_5d", "ITT_ITTInc_ret_5d", "Brent_Oil_FRED_ret_5d", "BA_ret_1d", "DHR_ret_1d", "SLB_Schlumberger_ret_5d", "ORCL_zscore_60d", "NWL_Newell_ret_20d", "PPL_PPL_ret_1d", "IWM_SmallCap_vol_20d", "HangSeng_HK_ret_5d", "SJM_JM_Smucker_ret_5d", "Core_PCE_zscore_60d", "EWQ_France_zscore_60d", "US30Y_Rate_ret_20d", "MS_MorganStanley_ret_5d"], "is_new": true}, {"model_id": "new_h1_STRESS_LightGBM_N25_t3", "algo": "LightGBM", "regime": "STRESS", "horizon": 1, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "IBEX_Spain_ret_20d", "T10Y2Y_Spread_ret_5d", "INTC_ret_5d", "TED_Spread_zscore_60d", "CCI_CrownCastle_vol_20d", "XLB_Materials_zscore_60d", "EWJ_Japan_vol_20d", "ENB_EnbridgeInc_ret_1d", "heston_ev_h3", "HD_ret_1d", "DHR_vol_20d", "ORCL_vol_20d", "US30Y_Rate_ret_20d", "PAYX_Paychex_ret_20d", "SBUX_zscore_60d", "EWY_Korea_zscore_60d", "Core_PCE_zscore_60d", "US1Y_Rate_ret_5d", "XOM_ret_20d", "LMT_LockheedMartin_ret_1d", "EQIX_Equinix_ret_5d", "AMZN_ret_5d", "LUV_SouthwestAir_ret_5d"], "is_new": true}, {"model_id": "new_h1_STRESS_LightGBM_N25_t4", "algo": "LightGBM", "regime": "STRESS", "horizon": 1, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "XLV_Health_zscore_60d", "US30Y_Rate_ret_20d", "EWA_Australia_ret_1d", "MSTR_Bitcoin3_ret_20d", "MRK_Merck_zscore_60d", "EQIX_Equinix_ret_5d", "AMD_ret_1d", "EWC_Canada_zscore_60d", "PAYX_Paychex_ret_20d", "LOW_Lowes_ret_5d", "TED_Spread_vol_20d", "EWA_Australia_zscore_60d", "3M_vol_20d", "VOD_Vodafone_zscore_60d", "IBEX_Spain_ret_20d", "IYM_BasicMaterials_ret_20d", "CPB_CampbellSoup_ret_20d", "MO_AltriaMG_ret_1d", "EWY_Korea_zscore_60d", "SJM_JM_Smucker_ret_5d", "AMD_ret_5d", "GE_ret_1d", "ASX_Australia_ret_5d"], "is_new": true}, {"model_id": "new_h1_STRESS_LightGBM_N25_t5", "algo": "LightGBM", "regime": "STRESS", "horizon": 1, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "AXP_Amex_ret_20d", "EWY_Korea_zscore_60d", "GILD_Gilead_ret_20d", "XOM_ret_20d", "BDX_Becton_Dickinson_ret_20d", "US3Y_Rate_ret_5d", "AORD_AUS_zscore_60d", "PPL_PPL_ret_1d", "XLF_Fin_vol_20d", "gjr_condvar_h1", "PCAR_PaccarInc_ret_5d", "SBUX_zscore_60d", "EOG_EOGResources_vol_20d", "QQQ_vol_20d", "TXN_vol_20d", "CPB_CampbellSoup_vol_20d", "ASX_Australia_vol_20d", "MO_AltriaMG_ret_1d", "heston_var_ev_h7", "EFFR_ret_1d", "EQIX_Equinix_ret_5d", "DE_Deere_ret_5d", "CI_Cigna_vol_20d"], "is_new": true}, {"model_id": "new_h1_STRESS_LightGBM_N25_t6", "algo": "LightGBM", "regime": "STRESS", "horizon": 1, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "heston_var_ev_h3", "BLK_BlackRock_zscore_60d", "INTC_ret_1d", "CPB_CampbellSoup_ret_20d", "TXN_vol_20d", "HD_ret_1d", "EWA_Australia_zscore_60d", "XLV_Health_zscore_60d", "AMGN_Amgen_ret_1d", "IYR_US_REIT2_zscore_60d", "INTC_ret_5d", "EOG_EOGResources_ret_5d", "VOD_Vodafone_zscore_60d", "LUV_SouthwestAir_ret_5d", "SPY_zscore_60d", "AMD_ret_5d", "Michigan_Sentiment_ret_20d", "NOC_Northrop_ret_20d", "vix_acceleration_1d", "BTI_BritishAmerican_ret_20d", "heston_var_ev_h7", "SJM_JM_Smucker_ret_5d", "DIS_vol_20d"], "is_new": true}, {"model_id": "new_h1_STRESS_LightGBM_N25_t7", "algo": "LightGBM", "regime": "STRESS", "horizon": 1, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "PAYX_Paychex_ret_20d", "AXP_Amex_ret_20d", "GILD_Gilead_ret_20d", "XLF_Fin_vol_20d", "EWC_Canada_zscore_60d", "SLB_Schlumberger_ret_1d", "NEE_NextEra_ret_20d", "EOG_EOGResources_vol_20d", "heston_ev_h3", "3M_ret_5d", "IBEX_Spain_ret_20d", "spx_vol_5d", "US30Y_Rate_ret_20d", "LMT_LockheedMartin_ret_1d", "Core_PCE_zscore_60d", "JNJ_ret_1d", "PLD_Prologis_ret_5d", "US3M_Rate_zscore_60d", "CLX_Clorox_vol_20d", "US3M_Rate_vol_20d", "PG_ret_20d", "WTI_Oil_FRED_zscore_60d", "Core_CPI_zscore_60d"], "is_new": true}, {"model_id": "new_h1_STRESS_LightGBM_N30_t0", "algo": "LightGBM", "regime": "STRESS", "horizon": 1, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "INTC_ret_5d", "VRP_ma5", "XOM_ret_20d", "AMT_AmericanTower_ret_1d", "EOG_EOGResources_ret_5d", "AMD_ret_1d", "EWL_Switzerland_vol_20d", "heston_var_ev_h3", "MRK_Merck_zscore_60d", "PFE_ret_1d", "CCI_CrownCastle_vol_20d", "EWY_Korea_ret_20d", "HangSeng_HK_vol_20d", "EWA_Australia_ret_1d", "DE_Deere_ret_5d", "IBEX_Spain_ret_20d", "BA_ret_1d", "MSTR_Bitcoin3_ret_20d", "HD_ret_5d", "EQIX_Equinix_ret_5d", "HD_ret_1d", "DHR_ret_1d", "US3M_Rate_vol_20d", "INTC_ret_1d", "NFCI_ret_5d", "EWM_Malaysia_ret_1d", "XLY_Disc_vol_20d", "GE_ret_1d"], "is_new": true}, {"model_id": "new_h1_STRESS_LightGBM_N30_t1", "algo": "LightGBM", "regime": "STRESS", "horizon": 1, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "ORCL_zscore_60d", "FedFunds_zscore_60d", "CMCSA_ret_1d", "SBUX_zscore_60d", "LMT_LockheedMartin_vol_20d", "US6M_Rate_ret_20d", "LUV_SouthwestAir_ret_5d", "LMT_LockheedMartin_ret_1d", "T10Y2Y_Spread_ret_5d", "PAYX_Paychex_zscore_60d", "gjr_condvar_h1", "EOG_EOGResources_ret_5d", "ASX_Australia_vol_20d", "US7Y_Rate_ret_20d", "MSTR_Bitcoin3_ret_5d", "EFFR_ret_1d", "TM_Telephone_ret_1d", "US3M_Rate_zscore_60d", "EWH_HongKong_ret_5d", "ORCL_vol_20d", "LLY_zscore_60d", "HD_ret_5d", "WTI_Oil_FRED_zscore_60d", "JNJ_ret_1d", "SJM_JM_Smucker_ret_5d", "US1Y_Rate_ret_20d", "GD_GeneralDynamics_zscore_60d", "spx_vol_5d"], "is_new": true}, {"model_id": "new_h1_STRESS_LightGBM_N30_t2", "algo": "LightGBM", "regime": "STRESS", "horizon": 1, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "PLD_Prologis_ret_5d", "EXC_Exelon_zscore_60d", "ASX_Australia_ret_5d", "Core_CPI_zscore_60d", "SPY_zscore_60d", "gjr_condvar_h1", "ES_Evergy_ret_1d", "QQQ_vol_20d", "DHR_ret_1d", "vix_mean_abs_ret_5d", "VOD_Vodafone_zscore_60d", "ITT_ITTInc_ret_5d", "MRK_Merck_zscore_60d", "US3M_Rate_vol_20d", "MSTR_Bitcoin3_ret_1d", "hmm_p_stress", "spx_vol_5d", "XOM_ret_20d", "CPB_CampbellSoup_vol_20d", "LOW_Lowes_ret_20d", "AMD_ret_1d", "EFFR_vol_20d", "NVDA_vol_20d", "heston_var_ev_h5", "IYR_US_REIT2_zscore_60d", "Industrial_Production_zscore_60d", "CPB_CampbellSoup_ret_5d", "heston_var_ev_h7"], "is_new": true}, {"model_id": "new_h1_STRESS_LightGBM_N30_t3", "algo": "LightGBM", "regime": "STRESS", "horizon": 1, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "TXN_vol_20d", "ES_Evergy_ret_1d", "AMT_AmericanTower_ret_1d", "ORCL_zscore_60d", "XLV_Health_zscore_60d", "PCAR_PaccarInc_ret_5d", "VRP_ma5", "US5Y_Rate_ret_5d", "ORCL_vol_20d", "M_Macys_vol_20d", "QQQ_vol_20d", "EWS_Singapore_ret_5d", "EWY_Korea_zscore_60d", "MRK_Merck_zscore_60d", "WTI_Oil_FRED_zscore_60d", "Industrial_Production_zscore_60d", "SBUX_ret_5d", "heston_ev_h3", "HangSeng_HK_ret_1d", "AVB_AvalonBay_zscore_60d", "EWM_Malaysia_zscore_60d", "SLB_Schlumberger_ret_1d", "CPB_CampbellSoup_zscore_60d", "NVDA_vol_20d", "Nikkei_Japan_vol_20d", "GD_GeneralDynamics_zscore_60d", "XLB_Materials_zscore_60d", "AORD_AUS_zscore_60d"], "is_new": true}, {"model_id": "new_h1_STRESS_LightGBM_N30_t4", "algo": "LightGBM", "regime": "STRESS", "horizon": 1, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "US3M_Rate_zscore_60d", "DHR_ret_1d", "DE_Deere_ret_5d", "IYR_US_REIT2_zscore_60d", "BA_ret_1d", "CMCSA_ret_1d", "IBEX_Spain_ret_20d", "DHR_vol_20d", "LLY_zscore_60d", "PFE_ret_1d", "MS_MorganStanley_zscore_60d", "ENB_EnbridgeInc_ret_1d", "heston_ev_h3", "3M_ret_5d", "DIS_vol_20d", "EWG_Germany_ret_20d", "EOG_EOGResources_vol_20d", "HangSeng_HK_ret_5d", "HangSeng_HK_vol_20d", "CCI_CrownCastle_vol_20d", "ES_Evergy_ret_1d", "EWQ_France_ret_20d", "EWG_Germany_vol_20d", "ORCL_vol_20d", "hmm_p_stress", "EWS_Singapore_ret_5d", "EFFR_ret_1d", "US1Y_Rate_ret_5d"], "is_new": true}, {"model_id": "new_h1_STRESS_LightGBM_N30_t5", "algo": "LightGBM", "regime": "STRESS", "horizon": 1, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "LLY_zscore_60d", "CPB_CampbellSoup_ret_5d", "PPL_PPL_ret_1d", "T_ret_1d", "MS_MorganStanley_ret_5d", "EWJ_Japan_vol_20d", "spx_vol_5d", "MSTR_Bitcoin3_ret_1d", "PG_ret_20d", "EWS_Singapore_ret_5d", "CPB_CampbellSoup_ret_20d", "MS_MorganStanley_ret_1d", "DAX_Germany_zscore_60d", "INTC_ret_5d", "DE_Deere_vol_20d", "M_Macys_vol_20d", "IWM_SmallCap_vol_20d", "HD_ret_20d", "NWL_Newell_ret_20d", "VVIX_ret_20d", "spx_abs_ret_max_5d", "TGT_Target_zscore_60d", "Brent_Oil_FRED_ret_20d", "vix_mean_abs_ret_5d", "SBUX_zscore_60d", "AMZN_ret_5d", "EWL_Switzerland_zscore_60d", "EQIX_Equinix_ret_5d"], "is_new": true}, {"model_id": "new_h1_STRESS_LightGBM_N30_t6", "algo": "LightGBM", "regime": "STRESS", "horizon": 1, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "SJM_JM_Smucker_ret_5d", "3M_ret_5d", "XLK_Tech_zscore_60d", "WTI_Oil_FRED_zscore_60d", "MS_MorganStanley_zscore_60d", "Brent_Oil_FRED_ret_5d", "heston_var_ev_h7", "EWM_Malaysia_vol_20d", "DHR_ret_1d", "SJM_JM_Smucker_ret_1d", "EWA_Australia_ret_1d", "AXP_Amex_ret_20d", "EWJ_Japan_vol_20d", "HangSeng_HK_ret_5d", "ORCL_vol_20d", "CMCSA_ret_1d", "Industrial_Production_zscore_60d", "MSTR_Bitcoin3_ret_5d", "EQR_Equity_ret_1d", "NWL_Newell_ret_20d", "DOW_Price_zscore_60d", "HD_ret_20d", "IBEX_Spain_ret_20d", "MS_MorganStanley_ret_1d", "SBUX_ret_5d", "ASX_Australia_ret_5d", "EXC_Exelon_zscore_60d", "NVDA_vol_20d"], "is_new": true}, {"model_id": "new_h1_STRESS_LightGBM_N30_t7", "algo": "LightGBM", "regime": "STRESS", "horizon": 1, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "PLD_Prologis_ret_5d", "CI_Cigna_vol_20d", "LLY_zscore_60d", "CPB_CampbellSoup_ret_5d", "spx_abs_ret_max_5d", "Michigan_Sentiment_ret_20d", "AMD_ret_5d", "EMR_Emerson_ret_20d", "INTC_ret_5d", "DAX_Germany_zscore_60d", "US5Y_Rate_ret_5d", "SLB_Schlumberger_ret_1d", "MSTR_Bitcoin3_ret_5d", "PCAR_PaccarInc_ret_5d", "AMD_ret_1d", "SBUX_vol_20d", "M_Macys_vol_20d", "NWL_Newell_ret_20d", "BA_ret_1d", "HangSeng_HK_ret_5d", "XLB_Materials_zscore_60d", "EXC_Exelon_zscore_60d", "Core_CPI_zscore_60d", "PAYX_Paychex_zscore_60d", "SJM_JM_Smucker_ret_5d", "MRK_Merck_zscore_60d", "QQQ_vol_20d", "IYR_US_REIT2_zscore_60d"], "is_new": true}, {"model_id": "new_h1_STRESS_GradientBoosting_N5_t0", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 1, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "NWL_Newell_ret_20d", "TGT_Target_zscore_60d", "MSTR_Bitcoin3_ret_5d"], "is_new": true}, {"model_id": "new_h1_STRESS_GradientBoosting_N5_t1", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 1, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "Industrial_Production_zscore_60d", "VRP_ma5", "EXC_Exelon_zscore_60d"], "is_new": true}, {"model_id": "new_h1_STRESS_GradientBoosting_N5_t2", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 1, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "PG_ret_20d", "HD_ret_20d", "AVB_AvalonBay_zscore_60d"], "is_new": true}, {"model_id": "new_h1_STRESS_GradientBoosting_N5_t3", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 1, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "AXP_Amex_vol_20d", "WTI_Oil_FRED_zscore_60d", "MO_AltriaMG_ret_1d"], "is_new": true}, {"model_id": "new_h1_STRESS_GradientBoosting_N5_t4", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 1, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "AMZN_ret_5d", "TM_Telephone_vol_20d", "GD_GeneralDynamics_zscore_60d"], "is_new": true}, {"model_id": "new_h1_STRESS_GradientBoosting_N5_t5", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 1, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "AMGN_Amgen_ret_1d", "PAYX_Paychex_vol_20d", "IWM_SmallCap_vol_20d"], "is_new": true}, {"model_id": "new_h1_STRESS_GradientBoosting_N5_t6", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 1, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "ASX_Australia_vol_20d", "TGT_Target_zscore_60d", "MS_MorganStanley_zscore_60d"], "is_new": true}, {"model_id": "new_h1_STRESS_GradientBoosting_N5_t7", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 1, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "DAX_Germany_zscore_60d", "DIS_vol_20d", "heston_ev_h3"], "is_new": true}, {"model_id": "new_h1_STRESS_GradientBoosting_N8_t0", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 1, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "PFE_ret_1d", "XLY_Disc_vol_20d", "EWS_Singapore_ret_5d", "TED_Spread_vol_20d", "US3M_Rate_zscore_60d", "AVB_AvalonBay_zscore_60d"], "is_new": true}, {"model_id": "new_h1_STRESS_GradientBoosting_N8_t1", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 1, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "DIS_vol_20d", "XLF_Fin_vol_20d", "heston_var_ev_h3", "CPB_CampbellSoup_ret_5d", "MSTR_Bitcoin3_ret_20d", "SO_SouthernCo_ret_5d"], "is_new": true}, {"model_id": "new_h1_STRESS_GradientBoosting_N8_t2", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 1, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "NWL_Newell_ret_20d", "M_Macys_vol_20d", "AMT_AmericanTower_ret_1d", "ES_Evergy_ret_1d", "EXC_Exelon_ret_1d", "spx_abs_ret_max_5d"], "is_new": true}, {"model_id": "new_h1_STRESS_GradientBoosting_N8_t3", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 1, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWA_Australia_zscore_60d", "US3Y_Rate_ret_5d", "EXC_Exelon_zscore_60d", "HangSeng_HK_ret_5d", "XLV_Health_zscore_60d", "SO_SouthernCo_ret_5d"], "is_new": true}, {"model_id": "new_h1_STRESS_GradientBoosting_N8_t4", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 1, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "US3Y_Rate_ret_5d", "QQQ_vol_20d", "EWL_Switzerland_vol_20d", "ORCL_vol_20d", "GILD_Gilead_ret_20d", "ES_Evergy_ret_1d"], "is_new": true}, {"model_id": "new_h1_STRESS_GradientBoosting_N8_t5", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 1, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "HD_ret_20d", "GE_ret_1d", "3M_vol_20d", "NEE_NextEra_ret_20d", "EWH_HongKong_ret_5d", "AMGN_Amgen_ret_1d"], "is_new": true}, {"model_id": "new_h1_STRESS_GradientBoosting_N8_t6", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 1, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "PG_ret_20d", "WTI_Oil_FRED_zscore_60d", "heston_ev_h3", "INTC_ret_5d", "CPB_CampbellSoup_ret_20d", "EWL_Switzerland_zscore_60d"], "is_new": true}, {"model_id": "new_h1_STRESS_GradientBoosting_N8_t7", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 1, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "BTI_BritishAmerican_ret_20d", "XLK_Tech_zscore_60d", "HD_ret_5d", "XLF_Fin_vol_20d", "PFE_ret_1d", "VOD_Vodafone_zscore_60d"], "is_new": true}, {"model_id": "new_h1_STRESS_GradientBoosting_N10_t0", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 1, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "US1Y_Rate_ret_5d", "AMD_ret_5d", "JNJ_ret_1d", "INTC_ret_5d", "LLY_zscore_60d", "PAYX_Paychex_ret_20d", "DOW_Price_zscore_60d", "PG_ret_20d"], "is_new": true}, {"model_id": "new_h1_STRESS_GradientBoosting_N10_t1", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 1, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "MS_MorganStanley_ret_1d", "EXC_Exelon_ret_1d", "Michigan_Sentiment_ret_20d", "ASX_Australia_ret_5d", "spx_vol_5d", "US3Y_Rate_ret_5d", "CCI_CrownCastle_vol_20d", "gjr_condvar_h1"], "is_new": true}, {"model_id": "new_h1_STRESS_GradientBoosting_N10_t2", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 1, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "TED_Spread_zscore_60d", "MO_AltriaMG_ret_1d", "HangSeng_HK_ret_5d", "HD_zscore_60d", "INTC_ret_1d", "PAYX_Paychex_zscore_60d", "PLD_Prologis_ret_5d", "AORD_AUS_zscore_60d"], "is_new": true}, {"model_id": "new_h1_STRESS_GradientBoosting_N10_t3", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 1, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "NOC_Northrop_ret_20d", "AMT_AmericanTower_ret_1d", "HD_ret_20d", "US1Y_Rate_ret_5d", "SBUX_zscore_60d", "EWQ_France_ret_20d", "MSTR_Bitcoin3_ret_1d", "TM_Telephone_ret_1d"], "is_new": true}, {"model_id": "new_h1_STRESS_GradientBoosting_N10_t4", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 1, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "MS_MorganStanley_zscore_60d", "EWL_Switzerland_vol_20d", "EWJ_Japan_vol_20d", "CTAS_Cintas_vol_20d", "US3Y_Rate_ret_5d", "HD_zscore_60d", "EQIX_Equinix_ret_5d", "EFFR_vol_20d"], "is_new": true}, {"model_id": "new_h1_STRESS_GradientBoosting_N10_t5", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 1, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWL_Switzerland_vol_20d", "NVDA_vol_20d", "WTI_Oil_FRED_zscore_60d", "HD_ret_1d", "INTC_ret_5d", "LOW_Lowes_ret_5d", "gjr_condvar_h1", "CPB_CampbellSoup_zscore_60d"], "is_new": true}, {"model_id": "new_h1_STRESS_GradientBoosting_N10_t6", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 1, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "Michigan_Sentiment_ret_20d", "SCHW_Schwab_ret_5d", "PG_ret_20d", "HD_ret_5d", "PAYX_Paychex_zscore_60d", "EFFR_vol_20d", "EQIX_Equinix_ret_5d", "M_Macys_vol_20d"], "is_new": true}, {"model_id": "new_h1_STRESS_GradientBoosting_N10_t7", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 1, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "spx_abs_ret_max_5d", "EOG_EOGResources_vol_20d", "CPB_CampbellSoup_ret_5d", "MSTR_Bitcoin3_ret_5d", "US6M_Rate_ret_20d", "NVDA_vol_20d", "NFCI_ret_5d", "EWA_Australia_ret_1d"], "is_new": true}, {"model_id": "new_h1_STRESS_GradientBoosting_N12_t0", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 1, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "XLK_Tech_zscore_60d", "XLY_Disc_vol_20d", "SJM_JM_Smucker_ret_5d", "CCI_CrownCastle_vol_20d", "WTI_Oil_FRED_zscore_60d", "US3Y_Rate_ret_5d", "NEE_NextEra_ret_20d", "AMT_AmericanTower_ret_1d", "IWM_SmallCap_vol_20d", "EWY_Korea_ret_20d"], "is_new": true}, {"model_id": "new_h1_STRESS_GradientBoosting_N12_t1", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 1, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "AMT_AmericanTower_ret_1d", "WTI_Oil_FRED_zscore_60d", "spx_abs_ret_max_5d", "MS_MorganStanley_ret_1d", "US1Y_Rate_ret_20d", "SLB_Schlumberger_ret_5d", "EWA_Australia_zscore_60d", "EMR_Emerson_ret_20d", "PFE_ret_1d", "MO_AltriaMG_ret_1d"], "is_new": true}, {"model_id": "new_h1_STRESS_GradientBoosting_N12_t2", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 1, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWG_Germany_ret_20d", "ITT_ITTInc_ret_5d", "DE_Deere_ret_5d", "HUM_Humana_ret_5d", "BTI_BritishAmerican_ret_20d", "AORD_AUS_zscore_60d", "EWG_Germany_vol_20d", "US5Y_Rate_ret_5d", "PG_ret_20d", "PFE_ret_1d"], "is_new": true}, {"model_id": "new_h1_STRESS_GradientBoosting_N12_t3", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 1, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "PG_ret_20d", "SJM_JM_Smucker_ret_5d", "heston_var_ev_h5", "CTAS_Cintas_vol_20d", "XLB_Materials_zscore_60d", "IYM_BasicMaterials_ret_20d", "EFFR_vol_20d", "EWS_Singapore_ret_5d", "BTI_BritishAmerican_ret_5d", "3M_vol_20d"], "is_new": true}, {"model_id": "new_h1_STRESS_GradientBoosting_N12_t4", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 1, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "MSTR_Bitcoin3_ret_5d", "Brent_Oil_FRED_ret_20d", "NOC_Northrop_ret_20d", "EXC_Exelon_zscore_60d", "TM_Telephone_ret_1d", "Core_PCE_zscore_60d", "XLF_Fin_vol_20d", "heston_var_ev_h7", "TGT_Target_zscore_60d", "AMZN_ret_5d"], "is_new": true}, {"model_id": "new_h1_STRESS_GradientBoosting_N12_t5", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 1, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "MS_MorganStanley_zscore_60d", "AVB_AvalonBay_zscore_60d", "LUV_SouthwestAir_ret_5d", "M_Macys_vol_20d", "SLB_Schlumberger_ret_5d", "gjr_condvar_h1", "hmm_p_stress", "ORCL_zscore_60d", "Nikkei_Japan_vol_20d", "DAX_Germany_vol_20d"], "is_new": true}, {"model_id": "new_h1_STRESS_GradientBoosting_N12_t6", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 1, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EOG_EOGResources_ret_5d", "LUV_SouthwestAir_ret_5d", "NFCI_ret_5d", "spx_momentum_3d", "Industrial_Production_zscore_60d", "DHR_vol_20d", "EWS_Singapore_ret_5d", "SBUX_ret_5d", "EWH_HongKong_ret_5d", "FedFunds_zscore_60d"], "is_new": true}, {"model_id": "new_h1_STRESS_GradientBoosting_N12_t7", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 1, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "SBUX_zscore_60d", "EWS_Singapore_ret_5d", "EXC_Exelon_zscore_60d", "JNJ_ret_1d", "HUM_Humana_ret_5d", "PAYX_Paychex_ret_20d", "IYM_BasicMaterials_ret_20d", "PAYX_Paychex_zscore_60d", "US3M_Rate_zscore_60d", "GILD_Gilead_ret_20d"], "is_new": true}, {"model_id": "new_h1_STRESS_GradientBoosting_N15_t0", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 1, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "T10Y2Y_Spread_ret_5d", "HD_ret_5d", "AMZN_ret_5d", "EWL_Switzerland_zscore_60d", "MO_AltriaMG_ret_1d", "US3M_Rate_vol_20d", "LOW_Lowes_ret_5d", "HD_zscore_60d", "HangSeng_HK_vol_20d", "EFFR_vol_20d", "US3M_Rate_zscore_60d", "heston_ev_h3", "QQQ_vol_20d"], "is_new": true}, {"model_id": "new_h1_STRESS_GradientBoosting_N15_t1", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 1, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "DAX_Germany_zscore_60d", "DIS_vol_20d", "XLF_Fin_vol_20d", "HD_zscore_60d", "MSTR_Bitcoin3_ret_5d", "EXC_Exelon_ret_1d", "heston_var_ev_h3", "EWM_Malaysia_vol_20d", "AORD_AUS_zscore_60d", "EWJ_Japan_vol_20d", "US7Y_Rate_ret_20d", "HangSeng_HK_ret_5d", "AMD_ret_5d"], "is_new": true}, {"model_id": "new_h1_STRESS_GradientBoosting_N15_t2", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 1, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "XLY_Disc_vol_20d", "EOG_EOGResources_ret_5d", "LLY_zscore_60d", "heston_var_ev_h5", "EFFR_ret_1d", "VOD_Vodafone_zscore_60d", "SJM_JM_Smucker_ret_5d", "INTC_ret_1d", "T10Y2Y_Spread_ret_5d", "MRK_Merck_zscore_60d", "MSTR_Bitcoin3_ret_1d", "SBUX_vol_20d", "heston_var_ev_h7"], "is_new": true}, {"model_id": "new_h1_STRESS_GradientBoosting_N15_t3", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 1, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "MSTR_Bitcoin3_ret_20d", "AMD_ret_5d", "vix_mean_abs_ret_5d", "BTI_BritishAmerican_ret_20d", "JNJ_ret_1d", "EWM_Malaysia_ret_1d", "IBEX_Spain_ret_20d", "EWG_Germany_vol_20d", "DOW_Price_zscore_60d", "HD_ret_1d", "XLF_Fin_vol_20d", "Core_PCE_zscore_60d", "hmm_p_stress"], "is_new": true}, {"model_id": "new_h1_STRESS_GradientBoosting_N15_t4", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 1, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "US3M_Rate_zscore_60d", "heston_var_ev_h3", "AMD_ret_5d", "TED_Spread_vol_20d", "EWM_Malaysia_vol_20d", "EWG_Germany_ret_20d", "LUV_SouthwestAir_ret_5d", "vix_acceleration_1d", "ASX_Australia_ret_5d", "M_Macys_vol_20d", "XOM_ret_20d", "CPB_CampbellSoup_ret_5d", "IYM_BasicMaterials_ret_20d"], "is_new": true}, {"model_id": "new_h1_STRESS_GradientBoosting_N15_t5", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 1, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "BDX_Becton_Dickinson_ret_20d", "PAYX_Paychex_vol_20d", "US3M_Rate_zscore_60d", "T10Y2Y_Spread_ret_5d", "EWY_Korea_ret_20d", "EWS_Singapore_ret_5d", "EXC_Exelon_ret_1d", "EQR_Equity_ret_1d", "hmm_p_stress", "EWM_Malaysia_vol_20d", "PCAR_PaccarInc_ret_5d", "EQIX_Equinix_ret_5d", "AVB_AvalonBay_zscore_60d"], "is_new": true}, {"model_id": "new_h1_STRESS_GradientBoosting_N15_t6", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 1, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWA_Australia_ret_1d", "BTI_BritishAmerican_ret_20d", "DE_Deere_ret_5d", "T10Y2Y_Spread_ret_5d", "HangSeng_HK_ret_5d", "BLK_BlackRock_zscore_60d", "EWH_HongKong_ret_5d", "EWM_Malaysia_vol_20d", "EWS_Singapore_ret_5d", "US7Y_Rate_ret_20d", "US6M_Rate_ret_20d", "Retail_Sales_zscore_60d", "PG_ret_20d"], "is_new": true}, {"model_id": "new_h1_STRESS_GradientBoosting_N15_t7", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 1, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWC_Canada_zscore_60d", "Retail_Sales_zscore_60d", "Core_CPI_zscore_60d", "PAYX_Paychex_ret_20d", "CI_Cigna_vol_20d", "QQQ_vol_20d", "CPB_CampbellSoup_ret_5d", "TM_Telephone_vol_20d", "VOD_Vodafone_zscore_60d", "EWL_Switzerland_vol_20d", "WTI_Oil_FRED_zscore_60d", "SBUX_vol_20d", "HangSeng_HK_ret_1d"], "is_new": true}, {"model_id": "new_h1_STRESS_GradientBoosting_N20_t0", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 1, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "PAYX_Paychex_ret_20d", "heston_var_ev_h5", "EWA_Australia_zscore_60d", "VOD_Vodafone_zscore_60d", "heston_ev_h3", "XLV_Health_zscore_60d", "PCAR_PaccarInc_ret_5d", "3M_ret_5d", "EWY_Korea_zscore_60d", "NEE_NextEra_ret_20d", "WTI_Oil_FRED_zscore_60d", "GE_ret_1d", "SCHW_Schwab_ret_5d", "gjr_condvar_h1", "EQIX_Equinix_ret_5d", "IYM_BasicMaterials_ret_20d", "SLB_Schlumberger_ret_1d", "SBUX_ret_5d"], "is_new": true}, {"model_id": "new_h1_STRESS_GradientBoosting_N20_t1", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 1, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "TGT_Target_zscore_60d", "EWY_Korea_ret_20d", "JNJ_ret_1d", "GILD_Gilead_ret_20d", "US30Y_Rate_ret_20d", "SJM_JM_Smucker_ret_5d", "IBEX_Spain_ret_20d", "XOM_ret_20d", "ORCL_zscore_60d", "CTAS_Cintas_vol_20d", "EQR_Equity_ret_1d", "SBUX_vol_20d", "3M_vol_20d", "INTC_ret_5d", "SLB_Schlumberger_ret_5d", "PPL_PPL_ret_1d", "SCHW_Schwab_ret_5d", "EMR_Emerson_ret_20d"], "is_new": true}, {"model_id": "new_h1_STRESS_GradientBoosting_N20_t2", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 1, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "CPB_CampbellSoup_ret_20d", "PAYX_Paychex_ret_20d", "JNJ_ret_1d", "heston_var_ev_h3", "TM_Telephone_ret_1d", "EQR_Equity_ret_1d", "EWA_Australia_zscore_60d", "CPB_CampbellSoup_zscore_60d", "EWM_Malaysia_zscore_60d", "EWJ_Japan_vol_20d", "HangSeng_HK_vol_20d", "SJM_JM_Smucker_ret_5d", "INTC_ret_5d", "TED_Spread_zscore_60d", "PPL_PPL_ret_1d", "CPB_CampbellSoup_vol_20d", "HangSeng_HK_ret_1d", "XLK_Tech_zscore_60d"], "is_new": true}, {"model_id": "new_h1_STRESS_GradientBoosting_N20_t3", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 1, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWA_Australia_zscore_60d", "MS_MorganStanley_zscore_60d", "EOG_EOGResources_ret_5d", "EFFR_vol_20d", "XLK_Tech_zscore_60d", "spx_vol_5d", "IWM_SmallCap_vol_20d", "SPY_zscore_60d", "QQQ_vol_20d", "spx_abs_ret_max_5d", "VVIX_ret_20d", "NOC_Northrop_ret_20d", "GD_GeneralDynamics_zscore_60d", "EWM_Malaysia_vol_20d", "FedFunds_zscore_60d", "PPL_PPL_ret_1d", "HangSeng_HK_ret_5d", "MO_AltriaMG_ret_1d"], "is_new": true}, {"model_id": "new_h1_STRESS_GradientBoosting_N20_t4", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 1, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "INTC_ret_1d", "PAYX_Paychex_zscore_60d", "HangSeng_HK_vol_20d", "NWL_Newell_ret_20d", "US3M_Rate_vol_20d", "MS_MorganStanley_ret_5d", "NOC_Northrop_ret_20d", "3M_vol_20d", "Michigan_Sentiment_ret_20d", "BDX_Becton_Dickinson_ret_20d", "NEE_NextEra_ret_20d", "LMT_LockheedMartin_vol_20d", "heston_var_ev_h7", "DHR_ret_1d", "PCAR_PaccarInc_ret_5d", "GILD_Gilead_ret_20d", "AMGN_Amgen_ret_1d", "EWG_Germany_ret_20d"], "is_new": true}, {"model_id": "new_h1_STRESS_GradientBoosting_N20_t5", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 1, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "AMZN_ret_5d", "ITT_ITTInc_ret_5d", "Nikkei_Japan_vol_20d", "MSTR_Bitcoin3_ret_20d", "IWM_SmallCap_vol_20d", "US1Y_Rate_ret_5d", "US30Y_Rate_ret_20d", "EWG_Germany_ret_20d", "EWL_Switzerland_vol_20d", "XLY_Disc_vol_20d", "SJM_JM_Smucker_ret_5d", "ORCL_zscore_60d", "GILD_Gilead_ret_20d", "CPB_CampbellSoup_zscore_60d", "Brent_Oil_FRED_ret_5d", "EWS_Singapore_ret_5d", "EMR_Emerson_ret_20d", "Retail_Sales_zscore_60d"], "is_new": true}, {"model_id": "new_h1_STRESS_GradientBoosting_N20_t6", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 1, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "SBUX_vol_20d", "EWY_Korea_ret_20d", "HD_ret_1d", "XLF_Fin_vol_20d", "VOD_Vodafone_zscore_60d", "MSTR_Bitcoin3_ret_1d", "BLK_BlackRock_zscore_60d", "CPB_CampbellSoup_ret_5d", "BDX_Becton_Dickinson_ret_20d", "heston_var_ev_h3", "JNJ_ret_1d", "GILD_Gilead_ret_20d", "TM_Telephone_ret_1d", "MO_AltriaMG_ret_1d", "ORCL_zscore_60d", "HD_ret_20d", "CTAS_Cintas_vol_20d", "PPL_PPL_ret_1d"], "is_new": true}, {"model_id": "new_h1_STRESS_GradientBoosting_N20_t7", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 1, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "DOW_Price_zscore_60d", "VOD_Vodafone_zscore_60d", "HangSeng_HK_ret_1d", "US3Y_Rate_ret_5d", "EWG_Germany_ret_20d", "US30Y_Rate_ret_20d", "CLX_Clorox_vol_20d", "ORCL_zscore_60d", "EWS_Singapore_ret_5d", "IWM_SmallCap_vol_20d", "TXN_vol_20d", "spx_vol_5d", "EXC_Exelon_zscore_60d", "US1Y_Rate_ret_5d", "TED_Spread_zscore_60d", "HD_zscore_60d", "AMT_AmericanTower_ret_1d", "DHR_ret_1d"], "is_new": true}, {"model_id": "new_h1_STRESS_GradientBoosting_N25_t0", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 1, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWJ_Japan_vol_20d", "ES_Evergy_ret_1d", "ITT_ITTInc_ret_5d", "PLD_Prologis_ret_5d", "EWM_Malaysia_vol_20d", "PFE_ret_1d", "LOW_Lowes_ret_5d", "IYR_US_REIT2_zscore_60d", "EWC_Canada_zscore_60d", "EXC_Exelon_zscore_60d", "EXC_Exelon_ret_1d", "XLB_Materials_zscore_60d", "AMT_AmericanTower_ret_1d", "DHR_ret_1d", "Core_PCE_zscore_60d", "NFCI_ret_5d", "Nikkei_Japan_vol_20d", "MS_MorganStanley_zscore_60d", "SBUX_vol_20d", "AXP_Amex_vol_20d", "TED_Spread_zscore_60d", "BLK_BlackRock_zscore_60d", "SO_SouthernCo_ret_5d"], "is_new": true}, {"model_id": "new_h1_STRESS_GradientBoosting_N25_t1", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 1, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "SLB_Schlumberger_ret_1d", "IWM_SmallCap_vol_20d", "TM_Telephone_ret_1d", "SBUX_zscore_60d", "HD_ret_20d", "US5Y_Rate_ret_5d", "IYR_US_REIT2_zscore_60d", "heston_ev_h3", "BTI_BritishAmerican_ret_5d", "Brent_Oil_FRED_ret_5d", "US3M_Rate_vol_20d", "CMCSA_ret_1d", "US1Y_Rate_ret_5d", "CTAS_Cintas_vol_20d", "TGT_Target_zscore_60d", "EFFR_vol_20d", "SO_SouthernCo_ret_5d", "CLX_Clorox_vol_20d", "US3Y_Rate_ret_5d", "PLD_Prologis_ret_5d", "EWA_Australia_ret_1d", "Brent_Oil_FRED_ret_20d", "Nikkei_Japan_zscore_60d"], "is_new": true}, {"model_id": "new_h1_STRESS_GradientBoosting_N25_t2", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 1, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "hmm_p_stress", "IYM_BasicMaterials_ret_20d", "Brent_Oil_FRED_ret_20d", "EQR_Equity_ret_1d", "heston_var_ev_h7", "US5Y_Rate_ret_5d", "Industrial_Production_zscore_60d", "PCAR_PaccarInc_ret_5d", "TED_Spread_zscore_60d", "Nikkei_Japan_zscore_60d", "AMT_AmericanTower_ret_1d", "M_Macys_vol_20d", "vix_mean_abs_ret_5d", "HUM_Humana_ret_5d", "3M_vol_20d", "MS_MorganStanley_ret_1d", "AMGN_Amgen_ret_1d", "AMD_ret_1d", "EWM_Malaysia_ret_1d", "ITT_ITTInc_ret_5d", "Nikkei_Japan_vol_20d", "DOW_Price_zscore_60d", "EXC_Exelon_zscore_60d"], "is_new": true}, {"model_id": "new_h1_STRESS_GradientBoosting_N25_t3", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 1, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "US5Y_Rate_ret_5d", "LMT_LockheedMartin_vol_20d", "vix_mean_abs_ret_5d", "EFFR_ret_1d", "EWA_Australia_ret_1d", "DIS_vol_20d", "EWG_Germany_vol_20d", "HangSeng_HK_vol_20d", "EQR_Equity_ret_1d", "CPB_CampbellSoup_ret_5d", "AMZN_ret_5d", "TGT_Target_zscore_60d", "XLB_Materials_zscore_60d", "AVB_AvalonBay_zscore_60d", "DAX_Germany_zscore_60d", "TM_Telephone_vol_20d", "LMT_LockheedMartin_ret_1d", "BTI_BritishAmerican_ret_5d", "IWM_SmallCap_vol_20d", "INTC_ret_1d", "EWM_Malaysia_vol_20d", "EWH_HongKong_ret_5d", "EWG_Germany_ret_20d"], "is_new": true}, {"model_id": "new_h1_STRESS_GradientBoosting_N25_t4", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 1, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWM_Malaysia_zscore_60d", "EWL_Switzerland_vol_20d", "LMT_LockheedMartin_vol_20d", "AMZN_ret_5d", "DHR_ret_1d", "ORCL_zscore_60d", "DAX_Germany_vol_20d", "MSTR_Bitcoin3_ret_1d", "spx_momentum_3d", "PAYX_Paychex_ret_20d", "EWH_HongKong_ret_5d", "MSTR_Bitcoin3_ret_20d", "AXP_Amex_vol_20d", "PAYX_Paychex_vol_20d", "hmm_p_stress", "M_Macys_vol_20d", "EWC_Canada_zscore_60d", "EWG_Germany_vol_20d", "XOM_ret_20d", "VRP_ma5", "heston_var_ev_h7", "CMCSA_ret_1d", "EOG_EOGResources_vol_20d"], "is_new": true}, {"model_id": "new_h1_STRESS_GradientBoosting_N25_t5", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 1, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "Core_PCE_zscore_60d", "SBUX_zscore_60d", "XLB_Materials_zscore_60d", "LOW_Lowes_ret_20d", "INTC_ret_5d", "NOC_Northrop_ret_20d", "EWA_Australia_ret_1d", "EWH_HongKong_ret_5d", "CMCSA_ret_1d", "SLB_Schlumberger_ret_1d", "LOW_Lowes_ret_5d", "Retail_Sales_zscore_60d", "EXC_Exelon_ret_1d", "SLB_Schlumberger_ret_5d", "CTAS_Cintas_vol_20d", "PCAR_PaccarInc_ret_5d", "AMGN_Amgen_ret_1d", "3M_vol_20d", "MSTR_Bitcoin3_ret_20d", "NVDA_vol_20d", "M_Macys_vol_20d", "CPB_CampbellSoup_ret_5d", "LMT_LockheedMartin_vol_20d"], "is_new": true}, {"model_id": "new_h1_STRESS_GradientBoosting_N25_t6", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 1, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "hmm_p_stress", "HangSeng_HK_ret_1d", "EWM_Malaysia_vol_20d", "CI_Cigna_vol_20d", "vix_mean_abs_ret_5d", "heston_var_ev_h5", "MS_MorganStanley_ret_1d", "PPL_PPL_ret_1d", "AMGN_Amgen_ret_1d", "GILD_Gilead_ret_20d", "ITT_ITTInc_ret_5d", "AXP_Amex_ret_20d", "DHR_vol_20d", "EWY_Korea_ret_20d", "EQIX_Equinix_ret_5d", "US3M_Rate_vol_20d", "AMD_ret_1d", "DAX_Germany_zscore_60d", "SPY_zscore_60d", "EFFR_ret_1d", "DHR_ret_1d", "spx_abs_ret_max_5d", "ASX_Australia_vol_20d"], "is_new": true}, {"model_id": "new_h1_STRESS_GradientBoosting_N25_t7", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 1, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "XLY_Disc_vol_20d", "NOC_Northrop_ret_20d", "EFFR_vol_20d", "NVDA_vol_20d", "EOG_EOGResources_vol_20d", "spx_abs_ret_max_5d", "SPY_zscore_60d", "DOW_Price_zscore_60d", "EQR_Equity_ret_1d", "LUV_SouthwestAir_ret_5d", "CTAS_Cintas_vol_20d", "EMR_Emerson_ret_20d", "AORD_AUS_zscore_60d", "SJM_JM_Smucker_ret_1d", "EWG_Germany_vol_20d", "HangSeng_HK_vol_20d", "Michigan_Sentiment_ret_20d", "MRK_Merck_zscore_60d", "FedFunds_zscore_60d", "ITT_ITTInc_ret_5d", "XOM_ret_1d", "AMZN_ret_5d", "IYM_BasicMaterials_ret_20d"], "is_new": true}, {"model_id": "new_h1_STRESS_GradientBoosting_N30_t0", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 1, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "NEE_NextEra_ret_20d", "AXP_Amex_ret_20d", "TM_Telephone_vol_20d", "MSTR_Bitcoin3_ret_1d", "WTI_Oil_FRED_zscore_60d", "TED_Spread_vol_20d", "EOG_EOGResources_ret_5d", "EFFR_vol_20d", "CPB_CampbellSoup_ret_5d", "AMD_ret_5d", "LMT_LockheedMartin_ret_1d", "EWM_Malaysia_ret_1d", "CMCSA_ret_1d", "NVDA_vol_20d", "BTI_BritishAmerican_ret_20d", "CTAS_Cintas_vol_20d", "QQQ_vol_20d", "PG_ret_20d", "NOC_Northrop_ret_20d", "TM_Telephone_ret_1d", "ORCL_zscore_60d", "DE_Deere_vol_20d", "3M_ret_5d", "IBEX_Spain_ret_20d", "XLF_Fin_vol_20d", "PPL_PPL_ret_1d", "EXC_Exelon_ret_1d", "DAX_Germany_zscore_60d"], "is_new": true}, {"model_id": "new_h1_STRESS_GradientBoosting_N30_t1", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 1, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "SCHW_Schwab_ret_5d", "US3Y_Rate_ret_5d", "EWC_Canada_zscore_60d", "DAX_Germany_zscore_60d", "EWL_Switzerland_zscore_60d", "SO_SouthernCo_ret_5d", "DE_Deere_ret_5d", "DHR_vol_20d", "EWM_Malaysia_ret_1d", "CI_Cigna_vol_20d", "CPB_CampbellSoup_ret_20d", "Michigan_Sentiment_ret_20d", "HD_ret_20d", "XLV_Health_zscore_60d", "CPB_CampbellSoup_vol_20d", "EWY_Korea_ret_20d", "heston_var_ev_h3", "EWY_Korea_zscore_60d", "ES_Evergy_ret_1d", "HangSeng_HK_ret_5d", "IYM_BasicMaterials_ret_20d", "PG_ret_20d", "T10Y2Y_Spread_ret_5d", "SPY_zscore_60d", "US5Y_Rate_ret_5d", "GD_GeneralDynamics_zscore_60d", "heston_var_ev_h7", "IWM_SmallCap_vol_20d"], "is_new": true}, {"model_id": "new_h1_STRESS_GradientBoosting_N30_t2", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 1, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "MSTR_Bitcoin3_ret_5d", "gjr_condvar_h1", "AXP_Amex_vol_20d", "EWC_Canada_zscore_60d", "EQR_Equity_ret_1d", "AMD_ret_5d", "DHR_vol_20d", "heston_ev_h3", "ASX_Australia_ret_5d", "EWQ_France_ret_20d", "EMR_Emerson_ret_20d", "MS_MorganStanley_zscore_60d", "ITT_ITTInc_ret_5d", "AMD_ret_1d", "US3Y_Rate_ret_5d", "SBUX_ret_5d", "XLK_Tech_zscore_60d", "LLY_zscore_60d", "EWA_Australia_ret_1d", "US5Y_Rate_ret_5d", "Michigan_Sentiment_ret_20d", "PAYX_Paychex_zscore_60d", "EWJ_Japan_vol_20d", "PCAR_PaccarInc_ret_5d", "FedFunds_zscore_60d", "TXN_vol_20d", "ORCL_zscore_60d", "Core_PCE_zscore_60d"], "is_new": true}, {"model_id": "new_h1_STRESS_GradientBoosting_N30_t3", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 1, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "ES_Evergy_ret_1d", "EXC_Exelon_ret_1d", "JNJ_ret_1d", "SBUX_vol_20d", "T_ret_1d", "PCAR_PaccarInc_ret_5d", "CPB_CampbellSoup_ret_20d", "spx_vol_5d", "SPY_zscore_60d", "FedFunds_zscore_60d", "BTI_BritishAmerican_ret_20d", "LUV_SouthwestAir_ret_5d", "Nikkei_Japan_vol_20d", "AMZN_ret_5d", "TXN_vol_20d", "DAX_Germany_zscore_60d", "heston_var_ev_h7", "EWM_Malaysia_vol_20d", "PFE_ret_1d", "CTAS_Cintas_vol_20d", "Core_PCE_zscore_60d", "spx_abs_ret_max_5d", "HUM_Humana_ret_5d", "XLK_Tech_zscore_60d", "LLY_zscore_60d", "HangSeng_HK_vol_20d", "PAYX_Paychex_zscore_60d", "ORCL_vol_20d"], "is_new": true}, {"model_id": "new_h1_STRESS_GradientBoosting_N30_t4", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 1, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "SLB_Schlumberger_ret_5d", "EWA_Australia_zscore_60d", "DHR_vol_20d", "EWL_Switzerland_zscore_60d", "AMGN_Amgen_ret_1d", "EWC_Canada_zscore_60d", "LMT_LockheedMartin_vol_20d", "TED_Spread_vol_20d", "WTI_Oil_FRED_zscore_60d", "DOW_Price_zscore_60d", "US3M_Rate_vol_20d", "TED_Spread_zscore_60d", "heston_var_ev_h5", "AXP_Amex_vol_20d", "TGT_Target_zscore_60d", "LOW_Lowes_ret_5d", "TM_Telephone_vol_20d", "EWM_Malaysia_vol_20d", "XOM_ret_20d", "ENB_EnbridgeInc_ret_1d", "MRK_Merck_zscore_60d", "EFFR_vol_20d", "CPB_CampbellSoup_ret_5d", "VVIX_ret_20d", "EWM_Malaysia_ret_1d", "heston_ev_h3", "XLF_Fin_vol_20d", "NWL_Newell_ret_20d"], "is_new": true}, {"model_id": "new_h1_STRESS_GradientBoosting_N30_t5", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 1, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "HUM_Humana_ret_5d", "MS_MorganStanley_ret_1d", "T10Y2Y_Spread_ret_5d", "NVDA_vol_20d", "QQQ_vol_20d", "PLD_Prologis_ret_5d", "ITT_ITTInc_ret_5d", "DE_Deere_vol_20d", "HangSeng_HK_ret_1d", "3M_ret_5d", "Brent_Oil_FRED_ret_5d", "EWY_Korea_zscore_60d", "US1Y_Rate_ret_5d", "IBEX_Spain_ret_20d", "EWH_HongKong_ret_5d", "XLK_Tech_zscore_60d", "VVIX_ret_20d", "BTI_BritishAmerican_ret_5d", "BLK_BlackRock_zscore_60d", "NFCI_ret_5d", "DHR_ret_1d", "DHR_vol_20d", "SBUX_ret_5d", "EMR_Emerson_ret_20d", "EWS_Singapore_ret_5d", "MS_MorganStanley_ret_5d", "EFFR_vol_20d", "spx_momentum_3d"], "is_new": true}, {"model_id": "new_h1_STRESS_GradientBoosting_N30_t6", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 1, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "Brent_Oil_FRED_ret_20d", "EWM_Malaysia_ret_1d", "GILD_Gilead_ret_20d", "DHR_ret_1d", "M_Macys_vol_20d", "DAX_Germany_vol_20d", "US7Y_Rate_ret_20d", "DOW_Price_zscore_60d", "EWG_Germany_vol_20d", "PAYX_Paychex_ret_20d", "DHR_vol_20d", "spx_vol_5d", "IWM_SmallCap_vol_20d", "AMD_ret_1d", "Core_PCE_zscore_60d", "T10Y2Y_Spread_ret_5d", "IBEX_Spain_ret_20d", "TED_Spread_vol_20d", "CPB_CampbellSoup_vol_20d", "PPL_PPL_ret_1d", "EWM_Malaysia_zscore_60d", "ORCL_zscore_60d", "HangSeng_HK_ret_1d", "hmm_p_stress", "TM_Telephone_vol_20d", "PCAR_PaccarInc_ret_5d", "EFFR_ret_1d", "vix_mean_abs_ret_5d"], "is_new": true}, {"model_id": "new_h1_STRESS_GradientBoosting_N30_t7", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 1, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "MSTR_Bitcoin3_ret_20d", "EXC_Exelon_ret_1d", "CTAS_Cintas_vol_20d", "XOM_ret_20d", "PLD_Prologis_ret_5d", "Nikkei_Japan_vol_20d", "US7Y_Rate_ret_20d", "EOG_EOGResources_ret_5d", "SBUX_vol_20d", "HD_zscore_60d", "EWM_Malaysia_ret_1d", "T10Y2Y_Spread_ret_5d", "CPB_CampbellSoup_vol_20d", "CLX_Clorox_vol_20d", "XLB_Materials_zscore_60d", "WTI_Oil_FRED_zscore_60d", "LOW_Lowes_ret_5d", "EWQ_France_zscore_60d", "SCHW_Schwab_ret_5d", "Nikkei_Japan_zscore_60d", "NVDA_vol_20d", "BDX_Becton_Dickinson_ret_20d", "XOM_ret_1d", "HD_ret_1d", "GILD_Gilead_ret_20d", "EWY_Korea_zscore_60d", "SLB_Schlumberger_ret_5d", "IYM_BasicMaterials_ret_20d"], "is_new": true}, {"model_id": "new_h1_STRESS_RandomForest_N5_t0", "algo": "RandomForest", "regime": "STRESS", "horizon": 1, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "AMD_ret_5d", "CPB_CampbellSoup_ret_5d", "SBUX_vol_20d"], "is_new": true}, {"model_id": "new_h1_STRESS_RandomForest_N5_t1", "algo": "RandomForest", "regime": "STRESS", "horizon": 1, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "vix_acceleration_1d", "GD_GeneralDynamics_zscore_60d", "CPB_CampbellSoup_vol_20d"], "is_new": true}, {"model_id": "new_h1_STRESS_RandomForest_N5_t2", "algo": "RandomForest", "regime": "STRESS", "horizon": 1, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "AVB_AvalonBay_zscore_60d", "XLB_Materials_zscore_60d", "CPB_CampbellSoup_zscore_60d"], "is_new": true}, {"model_id": "new_h1_STRESS_RandomForest_N5_t3", "algo": "RandomForest", "regime": "STRESS", "horizon": 1, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWQ_France_zscore_60d", "LOW_Lowes_ret_5d", "US6M_Rate_ret_20d"], "is_new": true}, {"model_id": "new_h1_STRESS_RandomForest_N5_t4", "algo": "RandomForest", "regime": "STRESS", "horizon": 1, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "LLY_zscore_60d", "heston_var_ev_h7", "ORCL_zscore_60d"], "is_new": true}, {"model_id": "new_h1_STRESS_RandomForest_N5_t5", "algo": "RandomForest", "regime": "STRESS", "horizon": 1, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "vix_mean_abs_ret_5d", "AMZN_ret_5d", "EWL_Switzerland_vol_20d"], "is_new": true}, {"model_id": "new_h1_STRESS_RandomForest_N5_t6", "algo": "RandomForest", "regime": "STRESS", "horizon": 1, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "CPB_CampbellSoup_ret_5d", "HD_zscore_60d", "NFCI_ret_5d"], "is_new": true}, {"model_id": "new_h1_STRESS_RandomForest_N5_t7", "algo": "RandomForest", "regime": "STRESS", "horizon": 1, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "LLY_zscore_60d", "LMT_LockheedMartin_vol_20d", "EWY_Korea_ret_20d"], "is_new": true}, {"model_id": "new_h1_STRESS_RandomForest_N8_t0", "algo": "RandomForest", "regime": "STRESS", "horizon": 1, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "PFE_ret_1d", "heston_ev_h3", "FedFunds_zscore_60d", "EMR_Emerson_ret_20d", "EWH_HongKong_ret_5d", "US1Y_Rate_ret_5d"], "is_new": true}, {"model_id": "new_h1_STRESS_RandomForest_N8_t1", "algo": "RandomForest", "regime": "STRESS", "horizon": 1, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EOG_EOGResources_vol_20d", "SJM_JM_Smucker_ret_1d", "CLX_Clorox_vol_20d", "IYR_US_REIT2_zscore_60d", "LMT_LockheedMartin_vol_20d", "XLV_Health_zscore_60d"], "is_new": true}, {"model_id": "new_h1_STRESS_RandomForest_N8_t2", "algo": "RandomForest", "regime": "STRESS", "horizon": 1, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "CPB_CampbellSoup_ret_5d", "CMCSA_ret_1d", "heston_var_ev_h3", "Michigan_Sentiment_ret_20d", "INTC_ret_1d", "XOM_ret_1d"], "is_new": true}, {"model_id": "new_h1_STRESS_RandomForest_N8_t3", "algo": "RandomForest", "regime": "STRESS", "horizon": 1, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "FedFunds_zscore_60d", "SBUX_vol_20d", "LMT_LockheedMartin_ret_1d", "Industrial_Production_zscore_60d", "ORCL_zscore_60d", "LOW_Lowes_ret_5d"], "is_new": true}, {"model_id": "new_h1_STRESS_RandomForest_N8_t4", "algo": "RandomForest", "regime": "STRESS", "horizon": 1, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "CTAS_Cintas_vol_20d", "EWA_Australia_ret_1d", "US6M_Rate_ret_20d", "IBEX_Spain_ret_20d", "SLB_Schlumberger_ret_1d", "heston_var_ev_h5"], "is_new": true}, {"model_id": "new_h1_STRESS_RandomForest_N8_t5", "algo": "RandomForest", "regime": "STRESS", "horizon": 1, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "XLY_Disc_vol_20d", "Brent_Oil_FRED_ret_5d", "EWC_Canada_zscore_60d", "CMCSA_ret_1d", "WTI_Oil_FRED_zscore_60d", "spx_vol_5d"], "is_new": true}, {"model_id": "new_h1_STRESS_RandomForest_N8_t6", "algo": "RandomForest", "regime": "STRESS", "horizon": 1, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "TGT_Target_zscore_60d", "LMT_LockheedMartin_vol_20d", "MS_MorganStanley_zscore_60d", "DE_Deere_vol_20d", "MSTR_Bitcoin3_ret_5d", "CPB_CampbellSoup_vol_20d"], "is_new": true}, {"model_id": "new_h1_STRESS_RandomForest_N8_t7", "algo": "RandomForest", "regime": "STRESS", "horizon": 1, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWM_Malaysia_ret_1d", "PAYX_Paychex_ret_20d", "AVB_AvalonBay_zscore_60d", "EWS_Singapore_ret_5d", "T_ret_1d", "EFFR_ret_1d"], "is_new": true}, {"model_id": "new_h1_STRESS_RandomForest_N10_t0", "algo": "RandomForest", "regime": "STRESS", "horizon": 1, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWM_Malaysia_ret_1d", "EWL_Switzerland_zscore_60d", "EWS_Singapore_ret_5d", "ORCL_zscore_60d", "heston_var_ev_h7", "EWL_Switzerland_vol_20d", "DIS_vol_20d", "AMD_ret_5d"], "is_new": true}, {"model_id": "new_h1_STRESS_RandomForest_N10_t1", "algo": "RandomForest", "regime": "STRESS", "horizon": 1, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "ORCL_zscore_60d", "NOC_Northrop_ret_20d", "US1Y_Rate_ret_5d", "US1Y_Rate_ret_20d", "US3M_Rate_zscore_60d", "EWA_Australia_zscore_60d", "NFCI_ret_5d", "ES_Evergy_ret_1d"], "is_new": true}, {"model_id": "new_h1_STRESS_RandomForest_N10_t2", "algo": "RandomForest", "regime": "STRESS", "horizon": 1, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "LUV_SouthwestAir_ret_5d", "FedFunds_zscore_60d", "Nikkei_Japan_zscore_60d", "MO_AltriaMG_ret_1d", "EWQ_France_ret_20d", "INTC_ret_5d", "TM_Telephone_vol_20d", "LOW_Lowes_ret_20d"], "is_new": true}, {"model_id": "new_h1_STRESS_RandomForest_N10_t3", "algo": "RandomForest", "regime": "STRESS", "horizon": 1, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "TM_Telephone_ret_1d", "US6M_Rate_ret_20d", "DAX_Germany_zscore_60d", "CPB_CampbellSoup_ret_20d", "SLB_Schlumberger_ret_1d", "QQQ_vol_20d", "INTC_ret_1d", "CPB_CampbellSoup_vol_20d"], "is_new": true}, {"model_id": "new_h1_STRESS_RandomForest_N10_t4", "algo": "RandomForest", "regime": "STRESS", "horizon": 1, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "PLD_Prologis_ret_5d", "GD_GeneralDynamics_zscore_60d", "US5Y_Rate_ret_5d", "BA_ret_1d", "PFE_ret_1d", "XLY_Disc_vol_20d", "ORCL_zscore_60d", "SJM_JM_Smucker_ret_1d"], "is_new": true}, {"model_id": "new_h1_STRESS_RandomForest_N10_t5", "algo": "RandomForest", "regime": "STRESS", "horizon": 1, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "AMT_AmericanTower_ret_1d", "US30Y_Rate_ret_20d", "PAYX_Paychex_vol_20d", "hmm_p_stress", "BDX_Becton_Dickinson_ret_20d", "LLY_zscore_60d", "Retail_Sales_zscore_60d", "DIS_vol_20d"], "is_new": true}, {"model_id": "new_h1_STRESS_RandomForest_N10_t6", "algo": "RandomForest", "regime": "STRESS", "horizon": 1, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "US3M_Rate_zscore_60d", "EWY_Korea_zscore_60d", "AORD_AUS_zscore_60d", "WTI_Oil_FRED_zscore_60d", "SBUX_zscore_60d", "Nikkei_Japan_zscore_60d", "DE_Deere_vol_20d", "MS_MorganStanley_zscore_60d"], "is_new": true}, {"model_id": "new_h1_STRESS_RandomForest_N10_t7", "algo": "RandomForest", "regime": "STRESS", "horizon": 1, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "IYM_BasicMaterials_ret_20d", "EWM_Malaysia_zscore_60d", "US3M_Rate_zscore_60d", "SLB_Schlumberger_ret_5d", "NEE_NextEra_ret_20d", "DHR_vol_20d", "TM_Telephone_vol_20d", "VOD_Vodafone_zscore_60d"], "is_new": true}, {"model_id": "new_h1_STRESS_RandomForest_N12_t0", "algo": "RandomForest", "regime": "STRESS", "horizon": 1, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "VOD_Vodafone_zscore_60d", "Core_PCE_zscore_60d", "Brent_Oil_FRED_ret_20d", "XLV_Health_zscore_60d", "BTI_BritishAmerican_ret_20d", "TED_Spread_zscore_60d", "NFCI_ret_5d", "EMR_Emerson_ret_20d", "EWA_Australia_zscore_60d", "EWY_Korea_ret_20d"], "is_new": true}, {"model_id": "new_h1_STRESS_RandomForest_N12_t1", "algo": "RandomForest", "regime": "STRESS", "horizon": 1, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "hmm_p_stress", "SCHW_Schwab_ret_5d", "Core_PCE_zscore_60d", "IYM_BasicMaterials_ret_20d", "CLX_Clorox_vol_20d", "CPB_CampbellSoup_ret_20d", "Michigan_Sentiment_ret_20d", "US3Y_Rate_ret_5d", "MS_MorganStanley_ret_1d", "NEE_NextEra_ret_20d"], "is_new": true}, {"model_id": "new_h1_STRESS_RandomForest_N12_t2", "algo": "RandomForest", "regime": "STRESS", "horizon": 1, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EQR_Equity_ret_1d", "CPB_CampbellSoup_ret_20d", "IBEX_Spain_ret_20d", "LMT_LockheedMartin_vol_20d", "EQIX_Equinix_ret_5d", "Nikkei_Japan_zscore_60d", "EWY_Korea_zscore_60d", "SJM_JM_Smucker_ret_1d", "CMCSA_ret_1d", "ES_Evergy_ret_1d"], "is_new": true}, {"model_id": "new_h1_STRESS_RandomForest_N12_t3", "algo": "RandomForest", "regime": "STRESS", "horizon": 1, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "LMT_LockheedMartin_ret_1d", "heston_var_ev_h7", "heston_ev_h3", "CCI_CrownCastle_vol_20d", "Industrial_Production_zscore_60d", "PAYX_Paychex_zscore_60d", "CPB_CampbellSoup_ret_20d", "TED_Spread_vol_20d", "EXC_Exelon_ret_1d", "AMT_AmericanTower_ret_1d"], "is_new": true}, {"model_id": "new_h1_STRESS_RandomForest_N12_t4", "algo": "RandomForest", "regime": "STRESS", "horizon": 1, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "CPB_CampbellSoup_ret_20d", "EOG_EOGResources_ret_5d", "3M_ret_5d", "TED_Spread_zscore_60d", "AMD_ret_5d", "3M_vol_20d", "IWM_SmallCap_vol_20d", "MSTR_Bitcoin3_ret_5d", "HangSeng_HK_vol_20d", "AORD_AUS_zscore_60d"], "is_new": true}, {"model_id": "new_h1_STRESS_RandomForest_N12_t5", "algo": "RandomForest", "regime": "STRESS", "horizon": 1, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWA_Australia_ret_1d", "heston_ev_h3", "XLY_Disc_vol_20d", "SJM_JM_Smucker_ret_1d", "EQR_Equity_ret_1d", "TM_Telephone_ret_1d", "EWL_Switzerland_vol_20d", "LLY_zscore_60d", "CPB_CampbellSoup_ret_20d", "EFFR_ret_1d"], "is_new": true}, {"model_id": "new_h1_STRESS_RandomForest_N12_t6", "algo": "RandomForest", "regime": "STRESS", "horizon": 1, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EQIX_Equinix_ret_5d", "SLB_Schlumberger_ret_1d", "DOW_Price_zscore_60d", "PAYX_Paychex_ret_20d", "AORD_AUS_zscore_60d", "XOM_ret_1d", "ASX_Australia_vol_20d", "TM_Telephone_ret_1d", "Core_PCE_zscore_60d", "IBEX_Spain_ret_20d"], "is_new": true}, {"model_id": "new_h1_STRESS_RandomForest_N12_t7", "algo": "RandomForest", "regime": "STRESS", "horizon": 1, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "VRP_ma5", "SPY_zscore_60d", "ENB_EnbridgeInc_ret_1d", "CPB_CampbellSoup_ret_5d", "SJM_JM_Smucker_ret_1d", "EWA_Australia_zscore_60d", "XLB_Materials_zscore_60d", "EFFR_ret_1d", "XLK_Tech_zscore_60d", "Industrial_Production_zscore_60d"], "is_new": true}, {"model_id": "new_h1_STRESS_RandomForest_N15_t0", "algo": "RandomForest", "regime": "STRESS", "horizon": 1, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "vix_mean_abs_ret_5d", "US1Y_Rate_ret_20d", "BTI_BritishAmerican_ret_20d", "Industrial_Production_zscore_60d", "EWY_Korea_ret_20d", "LUV_SouthwestAir_ret_5d", "DHR_vol_20d", "EWQ_France_zscore_60d", "SBUX_zscore_60d", "EWA_Australia_ret_1d", "Michigan_Sentiment_ret_20d", "AVB_AvalonBay_zscore_60d", "LOW_Lowes_ret_20d"], "is_new": true}, {"model_id": "new_h1_STRESS_RandomForest_N15_t1", "algo": "RandomForest", "regime": "STRESS", "horizon": 1, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "spx_vol_5d", "spx_abs_ret_max_5d", "ITT_ITTInc_ret_5d", "TED_Spread_vol_20d", "US3M_Rate_vol_20d", "US5Y_Rate_ret_5d", "TGT_Target_zscore_60d", "T10Y2Y_Spread_ret_5d", "MS_MorganStanley_ret_5d", "TXN_vol_20d", "heston_var_ev_h5", "EWG_Germany_ret_20d", "HD_ret_20d"], "is_new": true}, {"model_id": "new_h1_STRESS_RandomForest_N15_t2", "algo": "RandomForest", "regime": "STRESS", "horizon": 1, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "HD_zscore_60d", "XOM_ret_20d", "Nikkei_Japan_vol_20d", "Michigan_Sentiment_ret_20d", "ORCL_vol_20d", "MSTR_Bitcoin3_ret_1d", "CI_Cigna_vol_20d", "SBUX_zscore_60d", "AMZN_ret_5d", "CPB_CampbellSoup_vol_20d", "ITT_ITTInc_ret_5d", "US30Y_Rate_ret_20d", "SBUX_ret_5d"], "is_new": true}, {"model_id": "new_h1_STRESS_RandomForest_N15_t3", "algo": "RandomForest", "regime": "STRESS", "horizon": 1, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "HD_ret_5d", "BDX_Becton_Dickinson_ret_20d", "TED_Spread_zscore_60d", "AVB_AvalonBay_zscore_60d", "EWA_Australia_ret_1d", "CLX_Clorox_vol_20d", "HangSeng_HK_vol_20d", "VOD_Vodafone_zscore_60d", "US3Y_Rate_ret_5d", "TM_Telephone_vol_20d", "EOG_EOGResources_vol_20d", "heston_var_ev_h5", "INTC_ret_1d"], "is_new": true}, {"model_id": "new_h1_STRESS_RandomForest_N15_t4", "algo": "RandomForest", "regime": "STRESS", "horizon": 1, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "HangSeng_HK_ret_5d", "MSTR_Bitcoin3_ret_5d", "EWA_Australia_ret_1d", "LMT_LockheedMartin_vol_20d", "TM_Telephone_ret_1d", "BA_ret_1d", "ORCL_vol_20d", "SLB_Schlumberger_ret_5d", "TED_Spread_vol_20d", "EQIX_Equinix_ret_5d", "MO_AltriaMG_ret_1d", "AXP_Amex_ret_20d", "EWJ_Japan_vol_20d"], "is_new": true}, {"model_id": "new_h1_STRESS_RandomForest_N15_t5", "algo": "RandomForest", "regime": "STRESS", "horizon": 1, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "INTC_ret_5d", "Brent_Oil_FRED_ret_20d", "EWG_Germany_vol_20d", "SCHW_Schwab_ret_5d", "US3Y_Rate_ret_5d", "heston_var_ev_h3", "XLK_Tech_zscore_60d", "AMD_ret_5d", "AXP_Amex_ret_20d", "MSTR_Bitcoin3_ret_20d", "Core_CPI_zscore_60d", "HD_zscore_60d", "3M_vol_20d"], "is_new": true}, {"model_id": "new_h1_STRESS_RandomForest_N15_t6", "algo": "RandomForest", "regime": "STRESS", "horizon": 1, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWM_Malaysia_vol_20d", "MRK_Merck_zscore_60d", "M_Macys_vol_20d", "AXP_Amex_ret_20d", "EFFR_ret_1d", "DHR_vol_20d", "WTI_Oil_FRED_zscore_60d", "EWY_Korea_ret_20d", "XOM_ret_1d", "vix_mean_abs_ret_5d", "EXC_Exelon_zscore_60d", "NWL_Newell_ret_20d", "US3Y_Rate_ret_5d"], "is_new": true}, {"model_id": "new_h1_STRESS_RandomForest_N15_t7", "algo": "RandomForest", "regime": "STRESS", "horizon": 1, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "NEE_NextEra_ret_20d", "AMD_ret_1d", "NOC_Northrop_ret_20d", "NFCI_ret_5d", "MSTR_Bitcoin3_ret_1d", "HD_ret_1d", "EWA_Australia_ret_1d", "SBUX_vol_20d", "EWY_Korea_zscore_60d", "WTI_Oil_FRED_zscore_60d", "XOM_ret_20d", "MS_MorganStanley_zscore_60d", "AVB_AvalonBay_zscore_60d"], "is_new": true}, {"model_id": "new_h1_STRESS_RandomForest_N20_t0", "algo": "RandomForest", "regime": "STRESS", "horizon": 1, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "Brent_Oil_FRED_ret_20d", "heston_var_ev_h3", "Core_CPI_zscore_60d", "EFFR_vol_20d", "SBUX_zscore_60d", "BTI_BritishAmerican_ret_5d", "EQR_Equity_ret_1d", "TM_Telephone_ret_1d", "IWM_SmallCap_vol_20d", "ENB_EnbridgeInc_ret_1d", "EXC_Exelon_ret_1d", "MS_MorganStanley_zscore_60d", "HD_ret_20d", "HD_ret_5d", "US3M_Rate_zscore_60d", "EXC_Exelon_zscore_60d", "CPB_CampbellSoup_zscore_60d", "Core_PCE_zscore_60d"], "is_new": true}, {"model_id": "new_h1_STRESS_RandomForest_N20_t1", "algo": "RandomForest", "regime": "STRESS", "horizon": 1, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWG_Germany_ret_20d", "DAX_Germany_zscore_60d", "SPY_zscore_60d", "ASX_Australia_vol_20d", "PPL_PPL_ret_1d", "Retail_Sales_zscore_60d", "CPB_CampbellSoup_zscore_60d", "CTAS_Cintas_vol_20d", "EWM_Malaysia_vol_20d", "MS_MorganStanley_ret_1d", "ES_Evergy_ret_1d", "CI_Cigna_vol_20d", "BTI_BritishAmerican_ret_5d", "EWQ_France_zscore_60d", "ORCL_vol_20d", "XLV_Health_zscore_60d", "MO_AltriaMG_ret_1d", "ORCL_zscore_60d"], "is_new": true}, {"model_id": "new_h1_STRESS_RandomForest_N20_t2", "algo": "RandomForest", "regime": "STRESS", "horizon": 1, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "spx_vol_5d", "SJM_JM_Smucker_ret_5d", "EWA_Australia_ret_1d", "EWH_HongKong_ret_5d", "INTC_ret_5d", "AXP_Amex_ret_20d", "US3M_Rate_zscore_60d", "AMD_ret_5d", "EFFR_ret_1d", "JNJ_ret_1d", "EWY_Korea_zscore_60d", "EOG_EOGResources_vol_20d", "LMT_LockheedMartin_ret_1d", "PG_ret_20d", "EWG_Germany_ret_20d", "heston_var_ev_h5", "BDX_Becton_Dickinson_ret_20d", "MS_MorganStanley_ret_5d"], "is_new": true}, {"model_id": "new_h1_STRESS_RandomForest_N20_t3", "algo": "RandomForest", "regime": "STRESS", "horizon": 1, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWM_Malaysia_ret_1d", "ES_Evergy_ret_1d", "XOM_ret_20d", "EWA_Australia_zscore_60d", "IYR_US_REIT2_zscore_60d", "spx_momentum_3d", "ORCL_zscore_60d", "AMZN_ret_5d", "BTI_BritishAmerican_ret_5d", "EWS_Singapore_ret_5d", "XLB_Materials_zscore_60d", "TED_Spread_zscore_60d", "HD_ret_5d", "US1Y_Rate_ret_5d", "EMR_Emerson_ret_20d", "DHR_vol_20d", "T_ret_1d", "INTC_ret_5d"], "is_new": true}, {"model_id": "new_h1_STRESS_RandomForest_N20_t4", "algo": "RandomForest", "regime": "STRESS", "horizon": 1, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "SPY_zscore_60d", "TGT_Target_zscore_60d", "AMGN_Amgen_ret_1d", "XLY_Disc_vol_20d", "EOG_EOGResources_ret_5d", "ITT_ITTInc_ret_5d", "MO_AltriaMG_ret_1d", "ASX_Australia_vol_20d", "DHR_vol_20d", "CPB_CampbellSoup_vol_20d", "EWM_Malaysia_zscore_60d", "3M_vol_20d", "MSTR_Bitcoin3_ret_20d", "3M_ret_5d", "US3M_Rate_vol_20d", "BDX_Becton_Dickinson_ret_20d", "MS_MorganStanley_ret_1d", "US3M_Rate_zscore_60d"], "is_new": true}, {"model_id": "new_h1_STRESS_RandomForest_N20_t5", "algo": "RandomForest", "regime": "STRESS", "horizon": 1, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "AMT_AmericanTower_ret_1d", "hmm_p_stress", "QQQ_vol_20d", "US3Y_Rate_ret_5d", "heston_var_ev_h7", "MSTR_Bitcoin3_ret_5d", "HD_ret_20d", "XLV_Health_zscore_60d", "MS_MorganStanley_ret_1d", "PAYX_Paychex_zscore_60d", "EWQ_France_ret_20d", "SBUX_zscore_60d", "XLK_Tech_zscore_60d", "EWY_Korea_zscore_60d", "LLY_zscore_60d", "heston_ev_h3", "PCAR_PaccarInc_ret_5d", "Retail_Sales_zscore_60d"], "is_new": true}, {"model_id": "new_h1_STRESS_RandomForest_N20_t6", "algo": "RandomForest", "regime": "STRESS", "horizon": 1, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "DHR_vol_20d", "HD_ret_5d", "heston_var_ev_h3", "EOG_EOGResources_vol_20d", "MS_MorganStanley_ret_1d", "EXC_Exelon_ret_1d", "EMR_Emerson_ret_20d", "MO_AltriaMG_ret_1d", "AMT_AmericanTower_ret_1d", "Nikkei_Japan_zscore_60d", "HangSeng_HK_ret_1d", "AVB_AvalonBay_zscore_60d", "PFE_ret_1d", "NFCI_ret_5d", "TM_Telephone_vol_20d", "EWM_Malaysia_vol_20d", "M_Macys_vol_20d", "DAX_Germany_zscore_60d"], "is_new": true}, {"model_id": "new_h1_STRESS_RandomForest_N20_t7", "algo": "RandomForest", "regime": "STRESS", "horizon": 1, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "NVDA_vol_20d", "NOC_Northrop_ret_20d", "PFE_ret_1d", "PAYX_Paychex_zscore_60d", "PAYX_Paychex_ret_20d", "MS_MorganStanley_ret_1d", "PG_ret_20d", "LUV_SouthwestAir_ret_5d", "Industrial_Production_zscore_60d", "IWM_SmallCap_vol_20d", "vix_acceleration_1d", "M_Macys_vol_20d", "DE_Deere_vol_20d", "TED_Spread_zscore_60d", "EWG_Germany_vol_20d", "EWS_Singapore_ret_5d", "EWL_Switzerland_vol_20d", "EWA_Australia_ret_1d"], "is_new": true}, {"model_id": "new_h1_STRESS_RandomForest_N25_t0", "algo": "RandomForest", "regime": "STRESS", "horizon": 1, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "T_ret_1d", "NEE_NextEra_ret_20d", "MRK_Merck_zscore_60d", "heston_var_ev_h5", "HangSeng_HK_vol_20d", "EWJ_Japan_vol_20d", "MS_MorganStanley_ret_5d", "DHR_vol_20d", "PPL_PPL_ret_1d", "CPB_CampbellSoup_zscore_60d", "TM_Telephone_vol_20d", "PAYX_Paychex_zscore_60d", "HangSeng_HK_ret_1d", "NFCI_ret_5d", "BA_ret_1d", "EWG_Germany_ret_20d", "EWL_Switzerland_vol_20d", "Industrial_Production_zscore_60d", "US1Y_Rate_ret_20d", "EFFR_vol_20d", "MSTR_Bitcoin3_ret_5d", "SPY_zscore_60d", "Nikkei_Japan_zscore_60d"], "is_new": true}, {"model_id": "new_h1_STRESS_RandomForest_N25_t1", "algo": "RandomForest", "regime": "STRESS", "horizon": 1, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "MSTR_Bitcoin3_ret_5d", "US3M_Rate_zscore_60d", "EOG_EOGResources_vol_20d", "GE_ret_1d", "US3M_Rate_vol_20d", "EFFR_ret_1d", "heston_var_ev_h5", "M_Macys_vol_20d", "US5Y_Rate_ret_5d", "EWC_Canada_zscore_60d", "DOW_Price_zscore_60d", "NFCI_ret_5d", "INTC_ret_5d", "PLD_Prologis_ret_5d", "EWS_Singapore_ret_5d", "DE_Deere_ret_5d", "SBUX_vol_20d", "INTC_ret_1d", "TED_Spread_zscore_60d", "ASX_Australia_vol_20d", "CPB_CampbellSoup_ret_20d", "BTI_BritishAmerican_ret_20d", "CI_Cigna_vol_20d"], "is_new": true}, {"model_id": "new_h1_STRESS_RandomForest_N25_t2", "algo": "RandomForest", "regime": "STRESS", "horizon": 1, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "CMCSA_ret_1d", "INTC_ret_1d", "LOW_Lowes_ret_20d", "EWG_Germany_ret_20d", "DOW_Price_zscore_60d", "EWJ_Japan_vol_20d", "HangSeng_HK_vol_20d", "BTI_BritishAmerican_ret_20d", "LOW_Lowes_ret_5d", "XOM_ret_1d", "vix_acceleration_1d", "US7Y_Rate_ret_20d", "PAYX_Paychex_vol_20d", "QQQ_vol_20d", "Industrial_Production_zscore_60d", "EOG_EOGResources_vol_20d", "HD_zscore_60d", "HangSeng_HK_ret_5d", "CPB_CampbellSoup_ret_5d", "US30Y_Rate_ret_20d", "EFFR_vol_20d", "spx_abs_ret_max_5d", "EMR_Emerson_ret_20d"], "is_new": true}, {"model_id": "new_h1_STRESS_RandomForest_N25_t3", "algo": "RandomForest", "regime": "STRESS", "horizon": 1, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "US30Y_Rate_ret_20d", "US1Y_Rate_ret_20d", "XOM_ret_20d", "PPL_PPL_ret_1d", "HD_ret_1d", "EWY_Korea_ret_20d", "Core_CPI_zscore_60d", "DIS_vol_20d", "PAYX_Paychex_ret_20d", "CPB_CampbellSoup_ret_20d", "AVB_AvalonBay_zscore_60d", "DE_Deere_vol_20d", "MSTR_Bitcoin3_ret_20d", "QQQ_vol_20d", "XOM_ret_1d", "Retail_Sales_zscore_60d", "EOG_EOGResources_ret_5d", "AMT_AmericanTower_ret_1d", "VOD_Vodafone_zscore_60d", "LOW_Lowes_ret_5d", "ORCL_zscore_60d", "US7Y_Rate_ret_20d", "US3Y_Rate_ret_5d"], "is_new": true}, {"model_id": "new_h1_STRESS_RandomForest_N25_t4", "algo": "RandomForest", "regime": "STRESS", "horizon": 1, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "vix_mean_abs_ret_5d", "US3M_Rate_zscore_60d", "LLY_zscore_60d", "Industrial_Production_zscore_60d", "CI_Cigna_vol_20d", "MO_AltriaMG_ret_1d", "BTI_BritishAmerican_ret_5d", "EQIX_Equinix_ret_5d", "DE_Deere_vol_20d", "XOM_ret_1d", "HangSeng_HK_ret_1d", "MS_MorganStanley_ret_5d", "DOW_Price_zscore_60d", "CTAS_Cintas_vol_20d", "EWC_Canada_zscore_60d", "ITT_ITTInc_ret_5d", "HD_zscore_60d", "spx_abs_ret_max_5d", "LMT_LockheedMartin_vol_20d", "SCHW_Schwab_ret_5d", "AMGN_Amgen_ret_1d", "NOC_Northrop_ret_20d", "EOG_EOGResources_ret_5d"], "is_new": true}, {"model_id": "new_h1_STRESS_RandomForest_N25_t5", "algo": "RandomForest", "regime": "STRESS", "horizon": 1, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "T10Y2Y_Spread_ret_5d", "AMZN_ret_5d", "SBUX_zscore_60d", "INTC_ret_5d", "ENB_EnbridgeInc_ret_1d", "BA_ret_1d", "EWQ_France_zscore_60d", "HangSeng_HK_vol_20d", "FedFunds_zscore_60d", "PAYX_Paychex_vol_20d", "Nikkei_Japan_zscore_60d", "Industrial_Production_zscore_60d", "AVB_AvalonBay_zscore_60d", "GE_ret_1d", "PFE_ret_1d", "CCI_CrownCastle_vol_20d", "IYR_US_REIT2_zscore_60d", "US1Y_Rate_ret_20d", "ASX_Australia_ret_5d", "DE_Deere_ret_5d", "AMD_ret_1d", "US3Y_Rate_ret_5d", "AMGN_Amgen_ret_1d"], "is_new": true}, {"model_id": "new_h1_STRESS_RandomForest_N25_t6", "algo": "RandomForest", "regime": "STRESS", "horizon": 1, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "MSTR_Bitcoin3_ret_20d", "MS_MorganStanley_ret_1d", "ASX_Australia_vol_20d", "GE_ret_1d", "MSTR_Bitcoin3_ret_1d", "IYR_US_REIT2_zscore_60d", "LUV_SouthwestAir_ret_5d", "SO_SouthernCo_ret_5d", "NWL_Newell_ret_20d", "NEE_NextEra_ret_20d", "JNJ_ret_1d", "DE_Deere_ret_5d", "LLY_zscore_60d", "EXC_Exelon_zscore_60d", "US1Y_Rate_ret_20d", "EWA_Australia_zscore_60d", "SBUX_zscore_60d", "HD_ret_20d", "GD_GeneralDynamics_zscore_60d", "CI_Cigna_vol_20d", "PAYX_Paychex_zscore_60d", "VVIX_ret_20d", "INTC_ret_5d"], "is_new": true}, {"model_id": "new_h1_STRESS_RandomForest_N25_t7", "algo": "RandomForest", "regime": "STRESS", "horizon": 1, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "CPB_CampbellSoup_zscore_60d", "EWS_Singapore_ret_5d", "Brent_Oil_FRED_ret_5d", "EQIX_Equinix_ret_5d", "MRK_Merck_zscore_60d", "BTI_BritishAmerican_ret_20d", "EFFR_ret_1d", "US3Y_Rate_ret_5d", "EWQ_France_ret_20d", "CLX_Clorox_vol_20d", "spx_vol_5d", "AMD_ret_1d", "AMD_ret_5d", "JNJ_ret_1d", "Brent_Oil_FRED_ret_20d", "heston_var_ev_h7", "US7Y_Rate_ret_20d", "PFE_ret_1d", "IYM_BasicMaterials_ret_20d", "vix_mean_abs_ret_5d", "DAX_Germany_vol_20d", "BDX_Becton_Dickinson_ret_20d", "DHR_ret_1d"], "is_new": true}, {"model_id": "new_h1_STRESS_RandomForest_N30_t0", "algo": "RandomForest", "regime": "STRESS", "horizon": 1, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "CMCSA_ret_1d", "3M_ret_5d", "Nikkei_Japan_zscore_60d", "EWM_Malaysia_zscore_60d", "EWH_HongKong_ret_5d", "AXP_Amex_vol_20d", "CCI_CrownCastle_vol_20d", "MS_MorganStanley_ret_1d", "ITT_ITTInc_ret_5d", "XOM_ret_20d", "TED_Spread_zscore_60d", "EWA_Australia_zscore_60d", "QQQ_vol_20d", "EWC_Canada_zscore_60d", "SCHW_Schwab_ret_5d", "XLB_Materials_zscore_60d", "ES_Evergy_ret_1d", "PPL_PPL_ret_1d", "HD_ret_5d", "VRP_ma5", "BLK_BlackRock_zscore_60d", "PAYX_Paychex_ret_20d", "XLF_Fin_vol_20d", "SLB_Schlumberger_ret_1d", "3M_vol_20d", "heston_var_ev_h7", "EQIX_Equinix_ret_5d", "XLY_Disc_vol_20d"], "is_new": true}, {"model_id": "new_h1_STRESS_RandomForest_N30_t1", "algo": "RandomForest", "regime": "STRESS", "horizon": 1, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "Brent_Oil_FRED_ret_5d", "EWA_Australia_ret_1d", "ES_Evergy_ret_1d", "ITT_ITTInc_ret_5d", "EWM_Malaysia_vol_20d", "EMR_Emerson_ret_20d", "MS_MorganStanley_zscore_60d", "heston_ev_h3", "VRP_ma5", "HUM_Humana_ret_5d", "WTI_Oil_FRED_zscore_60d", "NFCI_ret_5d", "SLB_Schlumberger_ret_1d", "HD_ret_5d", "EWL_Switzerland_vol_20d", "Nikkei_Japan_vol_20d", "EWY_Korea_zscore_60d", "LLY_zscore_60d", "EWQ_France_zscore_60d", "TED_Spread_zscore_60d", "3M_vol_20d", "XLF_Fin_vol_20d", "HD_ret_20d", "hmm_p_stress", "BTI_BritishAmerican_ret_20d", "BLK_BlackRock_zscore_60d", "US5Y_Rate_ret_5d", "SCHW_Schwab_ret_5d"], "is_new": true}, {"model_id": "new_h1_STRESS_RandomForest_N30_t2", "algo": "RandomForest", "regime": "STRESS", "horizon": 1, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "PAYX_Paychex_vol_20d", "AMD_ret_5d", "CLX_Clorox_vol_20d", "INTC_ret_5d", "VOD_Vodafone_zscore_60d", "LOW_Lowes_ret_5d", "XLF_Fin_vol_20d", "BTI_BritishAmerican_ret_20d", "US3M_Rate_zscore_60d", "FedFunds_zscore_60d", "spx_momentum_3d", "PAYX_Paychex_zscore_60d", "LOW_Lowes_ret_20d", "Retail_Sales_zscore_60d", "MO_AltriaMG_ret_1d", "SJM_JM_Smucker_ret_1d", "HangSeng_HK_ret_1d", "SBUX_zscore_60d", "INTC_ret_1d", "TXN_vol_20d", "EWA_Australia_ret_1d", "EWM_Malaysia_vol_20d", "EOG_EOGResources_ret_5d", "AVB_AvalonBay_zscore_60d", "LMT_LockheedMartin_vol_20d", "NEE_NextEra_ret_20d", "PLD_Prologis_ret_5d", "PG_ret_20d"], "is_new": true}, {"model_id": "new_h1_STRESS_RandomForest_N30_t3", "algo": "RandomForest", "regime": "STRESS", "horizon": 1, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "PAYX_Paychex_zscore_60d", "EFFR_ret_1d", "DHR_vol_20d", "MSTR_Bitcoin3_ret_5d", "NVDA_vol_20d", "AMZN_ret_5d", "DHR_ret_1d", "INTC_ret_5d", "MSTR_Bitcoin3_ret_20d", "gjr_condvar_h1", "DOW_Price_zscore_60d", "M_Macys_vol_20d", "LUV_SouthwestAir_ret_5d", "AMD_ret_5d", "ORCL_vol_20d", "EOG_EOGResources_vol_20d", "heston_ev_h3", "VRP_ma5", "ES_Evergy_ret_1d", "EWL_Switzerland_vol_20d", "CPB_CampbellSoup_ret_20d", "AMGN_Amgen_ret_1d", "MS_MorganStanley_ret_5d", "PPL_PPL_ret_1d", "XLF_Fin_vol_20d", "GE_ret_1d", "Industrial_Production_zscore_60d", "EWC_Canada_zscore_60d"], "is_new": true}, {"model_id": "new_h1_STRESS_RandomForest_N30_t4", "algo": "RandomForest", "regime": "STRESS", "horizon": 1, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "HangSeng_HK_ret_5d", "T_ret_1d", "HD_ret_1d", "EWM_Malaysia_vol_20d", "EWA_Australia_ret_1d", "SBUX_vol_20d", "XLF_Fin_vol_20d", "SJM_JM_Smucker_ret_1d", "Nikkei_Japan_zscore_60d", "ORCL_vol_20d", "EFFR_vol_20d", "hmm_p_stress", "BLK_BlackRock_zscore_60d", "LMT_LockheedMartin_vol_20d", "CPB_CampbellSoup_zscore_60d", "BTI_BritishAmerican_ret_5d", "ITT_ITTInc_ret_5d", "Retail_Sales_zscore_60d", "HD_zscore_60d", "DHR_vol_20d", "EWQ_France_ret_20d", "SPY_zscore_60d", "SCHW_Schwab_ret_5d", "EWL_Switzerland_vol_20d", "NOC_Northrop_ret_20d", "SBUX_zscore_60d", "AMD_ret_5d", "HangSeng_HK_vol_20d"], "is_new": true}, {"model_id": "new_h1_STRESS_RandomForest_N30_t5", "algo": "RandomForest", "regime": "STRESS", "horizon": 1, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "XLY_Disc_vol_20d", "Nikkei_Japan_vol_20d", "XLK_Tech_zscore_60d", "DAX_Germany_vol_20d", "WTI_Oil_FRED_zscore_60d", "DE_Deere_ret_5d", "TED_Spread_zscore_60d", "HD_ret_20d", "JNJ_ret_1d", "MRK_Merck_zscore_60d", "DAX_Germany_zscore_60d", "MO_AltriaMG_ret_1d", "US3M_Rate_vol_20d", "QQQ_vol_20d", "BTI_BritishAmerican_ret_20d", "AMGN_Amgen_ret_1d", "PAYX_Paychex_vol_20d", "XOM_ret_20d", "NOC_Northrop_ret_20d", "EWY_Korea_ret_20d", "EWH_HongKong_ret_5d", "IWM_SmallCap_vol_20d", "EWM_Malaysia_zscore_60d", "INTC_ret_1d", "vix_mean_abs_ret_5d", "HD_ret_1d", "US1Y_Rate_ret_5d", "heston_var_ev_h3"], "is_new": true}, {"model_id": "new_h1_STRESS_RandomForest_N30_t6", "algo": "RandomForest", "regime": "STRESS", "horizon": 1, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "ITT_ITTInc_ret_5d", "gjr_condvar_h1", "US3Y_Rate_ret_5d", "EWM_Malaysia_vol_20d", "VRP_ma5", "XLK_Tech_zscore_60d", "EWQ_France_ret_20d", "EWY_Korea_ret_20d", "SBUX_zscore_60d", "US3M_Rate_vol_20d", "PPL_PPL_ret_1d", "AMGN_Amgen_ret_1d", "INTC_ret_1d", "SLB_Schlumberger_ret_5d", "CPB_CampbellSoup_ret_20d", "MSTR_Bitcoin3_ret_5d", "DIS_vol_20d", "vix_mean_abs_ret_5d", "CCI_CrownCastle_vol_20d", "TED_Spread_zscore_60d", "heston_var_ev_h3", "PG_ret_20d", "GD_GeneralDynamics_zscore_60d", "BTI_BritishAmerican_ret_5d", "PAYX_Paychex_zscore_60d", "MS_MorganStanley_ret_1d", "EWA_Australia_zscore_60d", "GILD_Gilead_ret_20d"], "is_new": true}, {"model_id": "new_h1_STRESS_RandomForest_N30_t7", "algo": "RandomForest", "regime": "STRESS", "horizon": 1, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "QQQ_vol_20d", "EWY_Korea_ret_20d", "EWM_Malaysia_ret_1d", "heston_ev_h3", "3M_ret_5d", "IWM_SmallCap_vol_20d", "LOW_Lowes_ret_20d", "XLY_Disc_vol_20d", "SBUX_ret_5d", "US30Y_Rate_ret_20d", "DIS_vol_20d", "GE_ret_1d", "INTC_ret_5d", "SBUX_vol_20d", "INTC_ret_1d", "XLB_Materials_zscore_60d", "TM_Telephone_ret_1d", "EWG_Germany_ret_20d", "EWA_Australia_ret_1d", "VVIX_ret_20d", "Core_PCE_zscore_60d", "HUM_Humana_ret_5d", "BLK_BlackRock_zscore_60d", "EWA_Australia_zscore_60d", "TED_Spread_vol_20d", "SLB_Schlumberger_ret_1d", "HD_ret_20d", "vix_acceleration_1d"], "is_new": true}, {"model_id": "new_h1_STRESS_LogisticRegression_N5_t0", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 1, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EXC_Exelon_zscore_60d", "EWG_Germany_vol_20d", "DHR_vol_20d"], "is_new": true}, {"model_id": "new_h1_STRESS_LogisticRegression_N5_t1", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 1, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "CCI_CrownCastle_vol_20d", "MO_AltriaMG_ret_1d", "PLD_Prologis_ret_5d"], "is_new": true}, {"model_id": "new_h1_STRESS_LogisticRegression_N5_t2", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 1, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "VVIX_ret_20d", "PAYX_Paychex_ret_20d", "INTC_ret_1d"], "is_new": true}, {"model_id": "new_h1_STRESS_LogisticRegression_N5_t3", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 1, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWC_Canada_zscore_60d", "XLF_Fin_vol_20d", "PCAR_PaccarInc_ret_5d"], "is_new": true}, {"model_id": "new_h1_STRESS_LogisticRegression_N5_t4", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 1, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "AMGN_Amgen_ret_1d", "DAX_Germany_vol_20d", "BA_ret_1d"], "is_new": true}, {"model_id": "new_h1_STRESS_LogisticRegression_N5_t5", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 1, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "INTC_ret_1d", "MSTR_Bitcoin3_ret_1d", "T_ret_1d"], "is_new": true}, {"model_id": "new_h1_STRESS_LogisticRegression_N5_t6", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 1, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EQIX_Equinix_ret_5d", "IWM_SmallCap_vol_20d", "XLK_Tech_zscore_60d"], "is_new": true}, {"model_id": "new_h1_STRESS_LogisticRegression_N5_t7", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 1, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "SLB_Schlumberger_ret_5d", "BDX_Becton_Dickinson_ret_20d", "DE_Deere_ret_5d"], "is_new": true}, {"model_id": "new_h1_STRESS_LogisticRegression_N8_t0", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 1, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "DOW_Price_zscore_60d", "CPB_CampbellSoup_vol_20d", "T10Y2Y_Spread_ret_5d", "EMR_Emerson_ret_20d", "EWQ_France_ret_20d", "CMCSA_ret_1d"], "is_new": true}, {"model_id": "new_h1_STRESS_LogisticRegression_N8_t1", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 1, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "NFCI_ret_5d", "JNJ_ret_1d", "BTI_BritishAmerican_ret_5d", "SCHW_Schwab_ret_5d", "M_Macys_vol_20d", "TXN_vol_20d"], "is_new": true}, {"model_id": "new_h1_STRESS_LogisticRegression_N8_t2", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 1, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "T_ret_1d", "CLX_Clorox_vol_20d", "EWH_HongKong_ret_5d", "PAYX_Paychex_vol_20d", "EFFR_vol_20d", "Core_CPI_zscore_60d"], "is_new": true}, {"model_id": "new_h1_STRESS_LogisticRegression_N8_t3", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 1, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "heston_var_ev_h7", "US1Y_Rate_ret_5d", "AMZN_ret_5d", "BA_ret_1d", "heston_var_ev_h5", "EWM_Malaysia_ret_1d"], "is_new": true}, {"model_id": "new_h1_STRESS_LogisticRegression_N8_t4", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 1, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWM_Malaysia_zscore_60d", "EFFR_ret_1d", "DHR_ret_1d", "SJM_JM_Smucker_ret_5d", "INTC_ret_5d", "ITT_ITTInc_ret_5d"], "is_new": true}, {"model_id": "new_h1_STRESS_LogisticRegression_N8_t5", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 1, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "SCHW_Schwab_ret_5d", "AMZN_ret_5d", "vix_mean_abs_ret_5d", "NOC_Northrop_ret_20d", "T_ret_1d", "BTI_BritishAmerican_ret_5d"], "is_new": true}, {"model_id": "new_h1_STRESS_LogisticRegression_N8_t6", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 1, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWM_Malaysia_vol_20d", "US7Y_Rate_ret_20d", "SBUX_zscore_60d", "3M_vol_20d", "LMT_LockheedMartin_ret_1d", "CCI_CrownCastle_vol_20d"], "is_new": true}, {"model_id": "new_h1_STRESS_LogisticRegression_N8_t7", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 1, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWJ_Japan_vol_20d", "DOW_Price_zscore_60d", "TM_Telephone_ret_1d", "3M_vol_20d", "XOM_ret_20d", "XOM_ret_1d"], "is_new": true}, {"model_id": "new_h1_STRESS_LogisticRegression_N10_t0", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 1, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EFFR_vol_20d", "EWG_Germany_ret_20d", "XLF_Fin_vol_20d", "EXC_Exelon_ret_1d", "IYR_US_REIT2_zscore_60d", "EWC_Canada_zscore_60d", "IWM_SmallCap_vol_20d", "EXC_Exelon_zscore_60d"], "is_new": true}, {"model_id": "new_h1_STRESS_LogisticRegression_N10_t1", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 1, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "AORD_AUS_zscore_60d", "PAYX_Paychex_vol_20d", "XLB_Materials_zscore_60d", "hmm_p_stress", "vix_mean_abs_ret_5d", "SO_SouthernCo_ret_5d", "Michigan_Sentiment_ret_20d", "EWG_Germany_vol_20d"], "is_new": true}, {"model_id": "new_h1_STRESS_LogisticRegression_N10_t2", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 1, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "CPB_CampbellSoup_zscore_60d", "EWG_Germany_vol_20d", "EWG_Germany_ret_20d", "US3M_Rate_zscore_60d", "HangSeng_HK_ret_1d", "EWJ_Japan_vol_20d", "VRP_ma5", "HangSeng_HK_ret_5d"], "is_new": true}, {"model_id": "new_h1_STRESS_LogisticRegression_N10_t3", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 1, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "US3M_Rate_zscore_60d", "EWY_Korea_zscore_60d", "EWL_Switzerland_vol_20d", "SLB_Schlumberger_ret_5d", "AMZN_ret_5d", "IBEX_Spain_ret_20d", "AMD_ret_5d", "EQR_Equity_ret_1d"], "is_new": true}, {"model_id": "new_h1_STRESS_LogisticRegression_N10_t4", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 1, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "IWM_SmallCap_vol_20d", "PFE_ret_1d", "MS_MorganStanley_ret_5d", "XLB_Materials_zscore_60d", "ENB_EnbridgeInc_ret_1d", "SBUX_zscore_60d", "Retail_Sales_zscore_60d", "US30Y_Rate_ret_20d"], "is_new": true}, {"model_id": "new_h1_STRESS_LogisticRegression_N10_t5", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 1, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "XLV_Health_zscore_60d", "CPB_CampbellSoup_zscore_60d", "3M_ret_5d", "AORD_AUS_zscore_60d", "XLY_Disc_vol_20d", "AXP_Amex_vol_20d", "AMD_ret_1d", "GILD_Gilead_ret_20d"], "is_new": true}, {"model_id": "new_h1_STRESS_LogisticRegression_N10_t6", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 1, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "US3M_Rate_zscore_60d", "EFFR_vol_20d", "CPB_CampbellSoup_ret_20d", "BA_ret_1d", "MO_AltriaMG_ret_1d", "EWL_Switzerland_vol_20d", "heston_var_ev_h7", "EWA_Australia_ret_1d"], "is_new": true}, {"model_id": "new_h1_STRESS_LogisticRegression_N10_t7", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 1, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "NOC_Northrop_ret_20d", "spx_abs_ret_max_5d", "AMT_AmericanTower_ret_1d", "HangSeng_HK_ret_1d", "ASX_Australia_ret_5d", "Core_PCE_zscore_60d", "HangSeng_HK_ret_5d", "DHR_vol_20d"], "is_new": true}, {"model_id": "new_h1_STRESS_LogisticRegression_N12_t0", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 1, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "heston_ev_h3", "AORD_AUS_zscore_60d", "QQQ_vol_20d", "EOG_EOGResources_vol_20d", "AMD_ret_5d", "CTAS_Cintas_vol_20d", "CLX_Clorox_vol_20d", "EWQ_France_ret_20d", "PCAR_PaccarInc_ret_5d", "US6M_Rate_ret_20d"], "is_new": true}, {"model_id": "new_h1_STRESS_LogisticRegression_N12_t1", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 1, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "US3M_Rate_vol_20d", "Retail_Sales_zscore_60d", "BTI_BritishAmerican_ret_20d", "EFFR_ret_1d", "EWL_Switzerland_vol_20d", "MS_MorganStanley_zscore_60d", "MSTR_Bitcoin3_ret_1d", "AORD_AUS_zscore_60d", "ASX_Australia_vol_20d", "SJM_JM_Smucker_ret_1d"], "is_new": true}, {"model_id": "new_h1_STRESS_LogisticRegression_N12_t2", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 1, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "AMD_ret_5d", "US5Y_Rate_ret_5d", "AXP_Amex_ret_20d", "spx_momentum_3d", "BTI_BritishAmerican_ret_20d", "HD_ret_1d", "HD_ret_20d", "Retail_Sales_zscore_60d", "EWG_Germany_ret_20d", "GD_GeneralDynamics_zscore_60d"], "is_new": true}, {"model_id": "new_h1_STRESS_LogisticRegression_N12_t3", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 1, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "PAYX_Paychex_vol_20d", "3M_ret_5d", "hmm_p_stress", "AXP_Amex_ret_20d", "US3M_Rate_vol_20d", "ORCL_vol_20d", "US30Y_Rate_ret_20d", "SBUX_zscore_60d", "EWM_Malaysia_zscore_60d", "AXP_Amex_vol_20d"], "is_new": true}, {"model_id": "new_h1_STRESS_LogisticRegression_N12_t4", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 1, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "PCAR_PaccarInc_ret_5d", "PPL_PPL_ret_1d", "TXN_vol_20d", "EMR_Emerson_ret_20d", "MRK_Merck_zscore_60d", "spx_momentum_3d", "US1Y_Rate_ret_20d", "TM_Telephone_vol_20d", "DIS_vol_20d", "ITT_ITTInc_ret_5d"], "is_new": true}, {"model_id": "new_h1_STRESS_LogisticRegression_N12_t5", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 1, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "US6M_Rate_ret_20d", "M_Macys_vol_20d", "MRK_Merck_zscore_60d", "PLD_Prologis_ret_5d", "LLY_zscore_60d", "HD_zscore_60d", "DIS_vol_20d", "SCHW_Schwab_ret_5d", "TM_Telephone_vol_20d", "NEE_NextEra_ret_20d"], "is_new": true}, {"model_id": "new_h1_STRESS_LogisticRegression_N12_t6", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 1, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "DHR_ret_1d", "IYR_US_REIT2_zscore_60d", "LOW_Lowes_ret_20d", "heston_var_ev_h7", "PLD_Prologis_ret_5d", "PAYX_Paychex_zscore_60d", "TM_Telephone_vol_20d", "ENB_EnbridgeInc_ret_1d", "AVB_AvalonBay_zscore_60d", "JNJ_ret_1d"], "is_new": true}, {"model_id": "new_h1_STRESS_LogisticRegression_N12_t7", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 1, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "US3Y_Rate_ret_5d", "DHR_ret_1d", "EWG_Germany_vol_20d", "MS_MorganStanley_zscore_60d", "CCI_CrownCastle_vol_20d", "NVDA_vol_20d", "CTAS_Cintas_vol_20d", "SJM_JM_Smucker_ret_5d", "TXN_vol_20d", "AVB_AvalonBay_zscore_60d"], "is_new": true}, {"model_id": "new_h1_STRESS_LogisticRegression_N15_t0", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 1, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "T_ret_1d", "vix_acceleration_1d", "ENB_EnbridgeInc_ret_1d", "EWM_Malaysia_vol_20d", "IYR_US_REIT2_zscore_60d", "SBUX_zscore_60d", "Core_CPI_zscore_60d", "IWM_SmallCap_vol_20d", "MRK_Merck_zscore_60d", "MS_MorganStanley_zscore_60d", "T10Y2Y_Spread_ret_5d", "spx_momentum_3d", "MS_MorganStanley_ret_5d"], "is_new": true}, {"model_id": "new_h1_STRESS_LogisticRegression_N15_t1", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 1, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWM_Malaysia_ret_1d", "heston_var_ev_h3", "heston_var_ev_h5", "PCAR_PaccarInc_ret_5d", "EFFR_ret_1d", "EWA_Australia_ret_1d", "3M_ret_5d", "SLB_Schlumberger_ret_1d", "CPB_CampbellSoup_zscore_60d", "SCHW_Schwab_ret_5d", "DHR_vol_20d", "Brent_Oil_FRED_ret_20d", "HangSeng_HK_vol_20d"], "is_new": true}, {"model_id": "new_h1_STRESS_LogisticRegression_N15_t2", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 1, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "TM_Telephone_vol_20d", "heston_var_ev_h3", "CPB_CampbellSoup_ret_20d", "EWA_Australia_ret_1d", "AMZN_ret_5d", "AMGN_Amgen_ret_1d", "AORD_AUS_zscore_60d", "HUM_Humana_ret_5d", "Core_PCE_zscore_60d", "EWH_HongKong_ret_5d", "ES_Evergy_ret_1d", "Brent_Oil_FRED_ret_20d", "BTI_BritishAmerican_ret_5d"], "is_new": true}, {"model_id": "new_h1_STRESS_LogisticRegression_N15_t3", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 1, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "LOW_Lowes_ret_20d", "spx_vol_5d", "Michigan_Sentiment_ret_20d", "AMD_ret_1d", "VVIX_ret_20d", "HangSeng_HK_ret_1d", "JNJ_ret_1d", "MS_MorganStanley_ret_1d", "MRK_Merck_zscore_60d", "EWH_HongKong_ret_5d", "HangSeng_HK_ret_5d", "Nikkei_Japan_vol_20d", "XLY_Disc_vol_20d"], "is_new": true}, {"model_id": "new_h1_STRESS_LogisticRegression_N15_t4", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 1, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "ASX_Australia_ret_5d", "SJM_JM_Smucker_ret_1d", "INTC_ret_5d", "LMT_LockheedMartin_ret_1d", "AORD_AUS_zscore_60d", "EXC_Exelon_ret_1d", "QQQ_vol_20d", "MSTR_Bitcoin3_ret_5d", "TM_Telephone_ret_1d", "CPB_CampbellSoup_ret_5d", "NWL_Newell_ret_20d", "VOD_Vodafone_zscore_60d", "XLY_Disc_vol_20d"], "is_new": true}, {"model_id": "new_h1_STRESS_LogisticRegression_N15_t5", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 1, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "ASX_Australia_vol_20d", "SBUX_vol_20d", "CLX_Clorox_vol_20d", "QQQ_vol_20d", "EWY_Korea_zscore_60d", "TXN_vol_20d", "CMCSA_ret_1d", "CI_Cigna_vol_20d", "EWS_Singapore_ret_5d", "gjr_condvar_h1", "CPB_CampbellSoup_ret_20d", "PCAR_PaccarInc_ret_5d", "MSTR_Bitcoin3_ret_20d"], "is_new": true}, {"model_id": "new_h1_STRESS_LogisticRegression_N15_t6", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 1, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "ASX_Australia_ret_5d", "ITT_ITTInc_ret_5d", "IBEX_Spain_ret_20d", "US1Y_Rate_ret_20d", "PAYX_Paychex_zscore_60d", "EWL_Switzerland_vol_20d", "XLV_Health_zscore_60d", "NFCI_ret_5d", "CPB_CampbellSoup_ret_20d", "AMD_ret_1d", "DAX_Germany_zscore_60d", "VRP_ma5", "AORD_AUS_zscore_60d"], "is_new": true}, {"model_id": "new_h1_STRESS_LogisticRegression_N15_t7", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 1, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "PG_ret_20d", "GILD_Gilead_ret_20d", "US3M_Rate_vol_20d", "MS_MorganStanley_ret_1d", "TM_Telephone_ret_1d", "spx_abs_ret_max_5d", "AXP_Amex_vol_20d", "EWY_Korea_zscore_60d", "US7Y_Rate_ret_20d", "Core_PCE_zscore_60d", "SPY_zscore_60d", "SBUX_zscore_60d", "EOG_EOGResources_vol_20d"], "is_new": true}, {"model_id": "new_h1_STRESS_LogisticRegression_N20_t0", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 1, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "HD_zscore_60d", "WTI_Oil_FRED_zscore_60d", "IYM_BasicMaterials_ret_20d", "Nikkei_Japan_vol_20d", "CTAS_Cintas_vol_20d", "EFFR_ret_1d", "Brent_Oil_FRED_ret_5d", "EWC_Canada_zscore_60d", "SBUX_ret_5d", "SBUX_vol_20d", "EWY_Korea_ret_20d", "3M_vol_20d", "MS_MorganStanley_ret_1d", "spx_abs_ret_max_5d", "EQIX_Equinix_ret_5d", "MSTR_Bitcoin3_ret_1d", "AXP_Amex_ret_20d", "HangSeng_HK_ret_5d"], "is_new": true}, {"model_id": "new_h1_STRESS_LogisticRegression_N20_t1", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 1, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWM_Malaysia_vol_20d", "PAYX_Paychex_ret_20d", "Core_CPI_zscore_60d", "CPB_CampbellSoup_ret_5d", "ENB_EnbridgeInc_ret_1d", "DHR_ret_1d", "DE_Deere_ret_5d", "XLB_Materials_zscore_60d", "US3M_Rate_vol_20d", "AMZN_ret_5d", "ORCL_zscore_60d", "SJM_JM_Smucker_ret_1d", "LUV_SouthwestAir_ret_5d", "Nikkei_Japan_zscore_60d", "AMT_AmericanTower_ret_1d", "TED_Spread_vol_20d", "MO_AltriaMG_ret_1d", "heston_ev_h3"], "is_new": true}, {"model_id": "new_h1_STRESS_LogisticRegression_N20_t2", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 1, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "ASX_Australia_ret_5d", "HD_ret_1d", "ORCL_vol_20d", "US1Y_Rate_ret_5d", "NWL_Newell_ret_20d", "BTI_BritishAmerican_ret_5d", "BTI_BritishAmerican_ret_20d", "CLX_Clorox_vol_20d", "FedFunds_zscore_60d", "vix_acceleration_1d", "LOW_Lowes_ret_20d", "QQQ_vol_20d", "M_Macys_vol_20d", "SLB_Schlumberger_ret_1d", "EWS_Singapore_ret_5d", "DHR_ret_1d", "Core_PCE_zscore_60d", "AMZN_ret_5d"], "is_new": true}, {"model_id": "new_h1_STRESS_LogisticRegression_N20_t3", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 1, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "PAYX_Paychex_vol_20d", "spx_momentum_3d", "TXN_vol_20d", "MRK_Merck_zscore_60d", "US1Y_Rate_ret_5d", "FedFunds_zscore_60d", "Retail_Sales_zscore_60d", "BTI_BritishAmerican_ret_5d", "ES_Evergy_ret_1d", "US7Y_Rate_ret_20d", "NFCI_ret_5d", "Core_CPI_zscore_60d", "US30Y_Rate_ret_20d", "AVB_AvalonBay_zscore_60d", "EWM_Malaysia_zscore_60d", "EWG_Germany_ret_20d", "DIS_vol_20d", "HangSeng_HK_ret_1d"], "is_new": true}, {"model_id": "new_h1_STRESS_LogisticRegression_N20_t4", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 1, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "BLK_BlackRock_zscore_60d", "SLB_Schlumberger_ret_5d", "AMD_ret_5d", "CTAS_Cintas_vol_20d", "TED_Spread_vol_20d", "vix_acceleration_1d", "GD_GeneralDynamics_zscore_60d", "US7Y_Rate_ret_20d", "TED_Spread_zscore_60d", "AMD_ret_1d", "HangSeng_HK_ret_5d", "DHR_ret_1d", "INTC_ret_5d", "EWS_Singapore_ret_5d", "Retail_Sales_zscore_60d", "EWJ_Japan_vol_20d", "EWA_Australia_ret_1d", "spx_vol_5d"], "is_new": true}, {"model_id": "new_h1_STRESS_LogisticRegression_N20_t5", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 1, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "CPB_CampbellSoup_ret_5d", "EOG_EOGResources_vol_20d", "MS_MorganStanley_ret_1d", "US1Y_Rate_ret_20d", "heston_ev_h3", "XOM_ret_20d", "TM_Telephone_vol_20d", "Brent_Oil_FRED_ret_20d", "DIS_vol_20d", "vix_acceleration_1d", "DAX_Germany_zscore_60d", "SLB_Schlumberger_ret_5d", "BTI_BritishAmerican_ret_5d", "Retail_Sales_zscore_60d", "SO_SouthernCo_ret_5d", "US3M_Rate_vol_20d", "ENB_EnbridgeInc_ret_1d", "EWY_Korea_ret_20d"], "is_new": true}, {"model_id": "new_h1_STRESS_LogisticRegression_N20_t6", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 1, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "FedFunds_zscore_60d", "NEE_NextEra_ret_20d", "BLK_BlackRock_zscore_60d", "Nikkei_Japan_zscore_60d", "AVB_AvalonBay_zscore_60d", "ITT_ITTInc_ret_5d", "MS_MorganStanley_zscore_60d", "AORD_AUS_zscore_60d", "EWA_Australia_zscore_60d", "vix_mean_abs_ret_5d", "MRK_Merck_zscore_60d", "MS_MorganStanley_ret_5d", "PAYX_Paychex_zscore_60d", "XLF_Fin_vol_20d", "EWM_Malaysia_ret_1d", "DAX_Germany_zscore_60d", "Retail_Sales_zscore_60d", "XLV_Health_zscore_60d"], "is_new": true}, {"model_id": "new_h1_STRESS_LogisticRegression_N20_t7", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 1, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "WTI_Oil_FRED_zscore_60d", "T10Y2Y_Spread_ret_5d", "HD_ret_5d", "SJM_JM_Smucker_ret_1d", "Brent_Oil_FRED_ret_5d", "Nikkei_Japan_zscore_60d", "AORD_AUS_zscore_60d", "NOC_Northrop_ret_20d", "BTI_BritishAmerican_ret_20d", "US30Y_Rate_ret_20d", "CI_Cigna_vol_20d", "DIS_vol_20d", "heston_var_ev_h5", "AVB_AvalonBay_zscore_60d", "MO_AltriaMG_ret_1d", "HD_ret_1d", "MS_MorganStanley_ret_5d", "PG_ret_20d"], "is_new": true}, {"model_id": "new_h1_STRESS_LogisticRegression_N25_t0", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 1, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "HangSeng_HK_vol_20d", "PPL_PPL_ret_1d", "SLB_Schlumberger_ret_5d", "SBUX_vol_20d", "MSTR_Bitcoin3_ret_5d", "VOD_Vodafone_zscore_60d", "US30Y_Rate_ret_20d", "XLB_Materials_zscore_60d", "WTI_Oil_FRED_zscore_60d", "US3Y_Rate_ret_5d", "DHR_ret_1d", "vix_acceleration_1d", "Industrial_Production_zscore_60d", "CMCSA_ret_1d", "SBUX_zscore_60d", "JNJ_ret_1d", "TGT_Target_zscore_60d", "SPY_zscore_60d", "BTI_BritishAmerican_ret_20d", "EWY_Korea_zscore_60d", "GD_GeneralDynamics_zscore_60d", "GE_ret_1d", "HD_ret_5d"], "is_new": true}, {"model_id": "new_h1_STRESS_LogisticRegression_N25_t1", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 1, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWQ_France_zscore_60d", "US3M_Rate_vol_20d", "EWA_Australia_ret_1d", "LMT_LockheedMartin_ret_1d", "HangSeng_HK_vol_20d", "EWA_Australia_zscore_60d", "IYR_US_REIT2_zscore_60d", "SPY_zscore_60d", "NFCI_ret_5d", "HD_ret_1d", "FedFunds_zscore_60d", "WTI_Oil_FRED_zscore_60d", "NWL_Newell_ret_20d", "spx_vol_5d", "XLK_Tech_zscore_60d", "DHR_ret_1d", "HUM_Humana_ret_5d", "XLY_Disc_vol_20d", "LOW_Lowes_ret_5d", "TM_Telephone_vol_20d", "US1Y_Rate_ret_20d", "ITT_ITTInc_ret_5d", "EWG_Germany_vol_20d"], "is_new": true}, {"model_id": "new_h1_STRESS_LogisticRegression_N25_t2", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 1, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "heston_var_ev_h3", "EWY_Korea_ret_20d", "EWG_Germany_vol_20d", "WTI_Oil_FRED_zscore_60d", "heston_ev_h3", "Industrial_Production_zscore_60d", "EQR_Equity_ret_1d", "SJM_JM_Smucker_ret_1d", "MRK_Merck_zscore_60d", "EWS_Singapore_ret_5d", "MS_MorganStanley_ret_5d", "SBUX_ret_5d", "GE_ret_1d", "US3M_Rate_zscore_60d", "PFE_ret_1d", "TM_Telephone_ret_1d", "JNJ_ret_1d", "DIS_vol_20d", "EXC_Exelon_ret_1d", "Brent_Oil_FRED_ret_20d", "EWY_Korea_zscore_60d", "DAX_Germany_vol_20d", "MSTR_Bitcoin3_ret_1d"], "is_new": true}, {"model_id": "new_h1_STRESS_LogisticRegression_N25_t3", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 1, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWA_Australia_ret_1d", "EXC_Exelon_ret_1d", "SCHW_Schwab_ret_5d", "PCAR_PaccarInc_ret_5d", "T_ret_1d", "US7Y_Rate_ret_20d", "GILD_Gilead_ret_20d", "SLB_Schlumberger_ret_5d", "US3M_Rate_zscore_60d", "NVDA_vol_20d", "EWC_Canada_zscore_60d", "GD_GeneralDynamics_zscore_60d", "BA_ret_1d", "Nikkei_Japan_vol_20d", "SBUX_zscore_60d", "VVIX_ret_20d", "CPB_CampbellSoup_zscore_60d", "QQQ_vol_20d", "ORCL_vol_20d", "MS_MorganStanley_ret_1d", "INTC_ret_1d", "Core_PCE_zscore_60d", "DHR_ret_1d"], "is_new": true}, {"model_id": "new_h1_STRESS_LogisticRegression_N25_t4", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 1, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "AVB_AvalonBay_zscore_60d", "BTI_BritishAmerican_ret_5d", "HD_ret_1d", "CPB_CampbellSoup_zscore_60d", "T_ret_1d", "TM_Telephone_ret_1d", "GD_GeneralDynamics_zscore_60d", "DAX_Germany_zscore_60d", "CPB_CampbellSoup_ret_20d", "NWL_Newell_ret_20d", "EWG_Germany_vol_20d", "vix_mean_abs_ret_5d", "US3M_Rate_zscore_60d", "BLK_BlackRock_zscore_60d", "AMD_ret_1d", "CTAS_Cintas_vol_20d", "Industrial_Production_zscore_60d", "GILD_Gilead_ret_20d", "HD_ret_20d", "Nikkei_Japan_zscore_60d", "EWQ_France_zscore_60d", "EWM_Malaysia_zscore_60d", "EWL_Switzerland_zscore_60d"], "is_new": true}, {"model_id": "new_h1_STRESS_LogisticRegression_N25_t5", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 1, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "DOW_Price_zscore_60d", "AMT_AmericanTower_ret_1d", "US1Y_Rate_ret_20d", "US7Y_Rate_ret_20d", "INTC_ret_5d", "EWL_Switzerland_zscore_60d", "EOG_EOGResources_ret_5d", "MSTR_Bitcoin3_ret_5d", "EWG_Germany_vol_20d", "TM_Telephone_vol_20d", "XLV_Health_zscore_60d", "AMGN_Amgen_ret_1d", "EMR_Emerson_ret_20d", "CMCSA_ret_1d", "US3Y_Rate_ret_5d", "GE_ret_1d", "SBUX_vol_20d", "DAX_Germany_vol_20d", "CPB_CampbellSoup_ret_20d", "EWY_Korea_ret_20d", "NEE_NextEra_ret_20d", "hmm_p_stress", "EWQ_France_zscore_60d"], "is_new": true}, {"model_id": "new_h1_STRESS_LogisticRegression_N25_t6", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 1, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "HD_ret_1d", "CPB_CampbellSoup_zscore_60d", "3M_ret_5d", "EWQ_France_zscore_60d", "HangSeng_HK_vol_20d", "ORCL_vol_20d", "US6M_Rate_ret_20d", "WTI_Oil_FRED_zscore_60d", "MSTR_Bitcoin3_ret_5d", "PAYX_Paychex_vol_20d", "EWJ_Japan_vol_20d", "EWM_Malaysia_zscore_60d", "CPB_CampbellSoup_ret_5d", "CCI_CrownCastle_vol_20d", "NEE_NextEra_ret_20d", "PAYX_Paychex_ret_20d", "XLK_Tech_zscore_60d", "DOW_Price_zscore_60d", "heston_var_ev_h3", "MSTR_Bitcoin3_ret_20d", "AMT_AmericanTower_ret_1d", "TM_Telephone_vol_20d", "GE_ret_1d"], "is_new": true}, {"model_id": "new_h1_STRESS_LogisticRegression_N25_t7", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 1, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "ITT_ITTInc_ret_5d", "AMT_AmericanTower_ret_1d", "SJM_JM_Smucker_ret_1d", "SLB_Schlumberger_ret_1d", "XLV_Health_zscore_60d", "MS_MorganStanley_ret_1d", "PAYX_Paychex_ret_20d", "XLF_Fin_vol_20d", "EWG_Germany_vol_20d", "MRK_Merck_zscore_60d", "DHR_vol_20d", "Brent_Oil_FRED_ret_5d", "WTI_Oil_FRED_zscore_60d", "MSTR_Bitcoin3_ret_1d", "EOG_EOGResources_ret_5d", "Michigan_Sentiment_ret_20d", "EWH_HongKong_ret_5d", "EWL_Switzerland_vol_20d", "INTC_ret_1d", "LMT_LockheedMartin_vol_20d", "XLK_Tech_zscore_60d", "GILD_Gilead_ret_20d", "EWY_Korea_ret_20d"], "is_new": true}, {"model_id": "new_h1_STRESS_LogisticRegression_N30_t0", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 1, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "WTI_Oil_FRED_zscore_60d", "PPL_PPL_ret_1d", "AMZN_ret_5d", "US3M_Rate_vol_20d", "EWS_Singapore_ret_5d", "heston_var_ev_h7", "BTI_BritishAmerican_ret_20d", "PAYX_Paychex_zscore_60d", "AMT_AmericanTower_ret_1d", "MS_MorganStanley_ret_1d", "US3Y_Rate_ret_5d", "LLY_zscore_60d", "LOW_Lowes_ret_5d", "MSTR_Bitcoin3_ret_20d", "heston_var_ev_h3", "SBUX_zscore_60d", "ORCL_vol_20d", "PAYX_Paychex_vol_20d", "NWL_Newell_ret_20d", "CLX_Clorox_vol_20d", "XLF_Fin_vol_20d", "EXC_Exelon_ret_1d", "LMT_LockheedMartin_ret_1d", "EWM_Malaysia_vol_20d", "GD_GeneralDynamics_zscore_60d", "LUV_SouthwestAir_ret_5d", "EXC_Exelon_zscore_60d", "Retail_Sales_zscore_60d"], "is_new": true}, {"model_id": "new_h1_STRESS_LogisticRegression_N30_t1", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 1, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "ASX_Australia_vol_20d", "MSTR_Bitcoin3_ret_20d", "LUV_SouthwestAir_ret_5d", "AXP_Amex_vol_20d", "EWM_Malaysia_vol_20d", "CMCSA_ret_1d", "EFFR_vol_20d", "hmm_p_stress", "XLV_Health_zscore_60d", "US1Y_Rate_ret_20d", "spx_momentum_3d", "EWY_Korea_zscore_60d", "CCI_CrownCastle_vol_20d", "TED_Spread_vol_20d", "EXC_Exelon_ret_1d", "PAYX_Paychex_zscore_60d", "DOW_Price_zscore_60d", "HUM_Humana_ret_5d", "DHR_vol_20d", "ORCL_vol_20d", "HD_ret_1d", "ES_Evergy_ret_1d", "spx_abs_ret_max_5d", "NEE_NextEra_ret_20d", "IBEX_Spain_ret_20d", "ENB_EnbridgeInc_ret_1d", "T10Y2Y_Spread_ret_5d", "BTI_BritishAmerican_ret_5d"], "is_new": true}, {"model_id": "new_h1_STRESS_LogisticRegression_N30_t2", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 1, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "HD_ret_5d", "T_ret_1d", "PPL_PPL_ret_1d", "NWL_Newell_ret_20d", "ENB_EnbridgeInc_ret_1d", "EWQ_France_ret_20d", "AMGN_Amgen_ret_1d", "EWC_Canada_zscore_60d", "XLB_Materials_zscore_60d", "Brent_Oil_FRED_ret_5d", "BTI_BritishAmerican_ret_5d", "CPB_CampbellSoup_ret_5d", "GD_GeneralDynamics_zscore_60d", "EQIX_Equinix_ret_5d", "TGT_Target_zscore_60d", "DAX_Germany_vol_20d", "PG_ret_20d", "US1Y_Rate_ret_20d", "MO_AltriaMG_ret_1d", "XLY_Disc_vol_20d", "EFFR_ret_1d", "XLV_Health_zscore_60d", "ES_Evergy_ret_1d", "EWS_Singapore_ret_5d", "AMD_ret_1d", "IWM_SmallCap_vol_20d", "EOG_EOGResources_ret_5d", "GE_ret_1d"], "is_new": true}, {"model_id": "new_h1_STRESS_LogisticRegression_N30_t3", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 1, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "DE_Deere_vol_20d", "ENB_EnbridgeInc_ret_1d", "AXP_Amex_vol_20d", "PPL_PPL_ret_1d", "EFFR_ret_1d", "DE_Deere_ret_5d", "EWQ_France_zscore_60d", "XLF_Fin_vol_20d", "MO_AltriaMG_ret_1d", "US3Y_Rate_ret_5d", "HD_ret_1d", "PG_ret_20d", "EWS_Singapore_ret_5d", "PCAR_PaccarInc_ret_5d", "EOG_EOGResources_vol_20d", "CTAS_Cintas_vol_20d", "EWY_Korea_ret_20d", "EMR_Emerson_ret_20d", "AVB_AvalonBay_zscore_60d", "BTI_BritishAmerican_ret_5d", "MSTR_Bitcoin3_ret_1d", "vix_mean_abs_ret_5d", "3M_ret_5d", "HD_ret_5d", "PFE_ret_1d", "TED_Spread_zscore_60d", "SBUX_zscore_60d", "MS_MorganStanley_zscore_60d"], "is_new": true}, {"model_id": "new_h1_STRESS_LogisticRegression_N30_t4", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 1, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "XLF_Fin_vol_20d", "US5Y_Rate_ret_5d", "Michigan_Sentiment_ret_20d", "EWY_Korea_zscore_60d", "TED_Spread_vol_20d", "NEE_NextEra_ret_20d", "ENB_EnbridgeInc_ret_1d", "SJM_JM_Smucker_ret_5d", "SJM_JM_Smucker_ret_1d", "DAX_Germany_zscore_60d", "ASX_Australia_vol_20d", "EWG_Germany_vol_20d", "Brent_Oil_FRED_ret_20d", "WTI_Oil_FRED_zscore_60d", "MO_AltriaMG_ret_1d", "HangSeng_HK_ret_1d", "Core_CPI_zscore_60d", "XLB_Materials_zscore_60d", "NFCI_ret_5d", "MSTR_Bitcoin3_ret_20d", "MSTR_Bitcoin3_ret_1d", "XLK_Tech_zscore_60d", "MRK_Merck_zscore_60d", "AMZN_ret_5d", "INTC_ret_1d", "SBUX_vol_20d", "EWC_Canada_zscore_60d", "AMGN_Amgen_ret_1d"], "is_new": true}, {"model_id": "new_h1_STRESS_LogisticRegression_N30_t5", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 1, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "VRP_ma5", "DHR_ret_1d", "hmm_p_stress", "HangSeng_HK_ret_5d", "MS_MorganStanley_zscore_60d", "EWS_Singapore_ret_5d", "DHR_vol_20d", "AMGN_Amgen_ret_1d", "EWM_Malaysia_vol_20d", "ORCL_zscore_60d", "EWY_Korea_zscore_60d", "SBUX_vol_20d", "M_Macys_vol_20d", "PLD_Prologis_ret_5d", "XLY_Disc_vol_20d", "MRK_Merck_zscore_60d", "EWG_Germany_vol_20d", "PG_ret_20d", "DE_Deere_ret_5d", "NOC_Northrop_ret_20d", "BA_ret_1d", "spx_vol_5d", "DIS_vol_20d", "CPB_CampbellSoup_ret_5d", "HUM_Humana_ret_5d", "XLK_Tech_zscore_60d", "vix_mean_abs_ret_5d", "LMT_LockheedMartin_ret_1d"], "is_new": true}, {"model_id": "new_h1_STRESS_LogisticRegression_N30_t6", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 1, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "WTI_Oil_FRED_zscore_60d", "HUM_Humana_ret_5d", "LMT_LockheedMartin_vol_20d", "SO_SouthernCo_ret_5d", "US1Y_Rate_ret_20d", "Retail_Sales_zscore_60d", "AVB_AvalonBay_zscore_60d", "hmm_p_stress", "BTI_BritishAmerican_ret_5d", "spx_vol_5d", "IYM_BasicMaterials_ret_20d", "EWA_Australia_zscore_60d", "IBEX_Spain_ret_20d", "SLB_Schlumberger_ret_1d", "ENB_EnbridgeInc_ret_1d", "AMD_ret_1d", "CPB_CampbellSoup_ret_5d", "DIS_vol_20d", "US30Y_Rate_ret_20d", "GE_ret_1d", "BLK_BlackRock_zscore_60d", "AMZN_ret_5d", "SLB_Schlumberger_ret_5d", "EWL_Switzerland_vol_20d", "T10Y2Y_Spread_ret_5d", "MRK_Merck_zscore_60d", "VOD_Vodafone_zscore_60d", "PAYX_Paychex_vol_20d"], "is_new": true}, {"model_id": "new_h1_STRESS_LogisticRegression_N30_t7", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 1, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "XOM_ret_1d", "IWM_SmallCap_vol_20d", "HD_ret_20d", "MO_AltriaMG_ret_1d", "BA_ret_1d", "SPY_zscore_60d", "US7Y_Rate_ret_20d", "DAX_Germany_vol_20d", "ORCL_zscore_60d", "SBUX_vol_20d", "Nikkei_Japan_zscore_60d", "SO_SouthernCo_ret_5d", "ASX_Australia_vol_20d", "T_ret_1d", "US3M_Rate_vol_20d", "SLB_Schlumberger_ret_1d", "BTI_BritishAmerican_ret_5d", "EFFR_ret_1d", "MSTR_Bitcoin3_ret_1d", "IYM_BasicMaterials_ret_20d", "heston_var_ev_h3", "GD_GeneralDynamics_zscore_60d", "DHR_vol_20d", "heston_var_ev_h7", "EWC_Canada_zscore_60d", "CMCSA_ret_1d", "MSTR_Bitcoin3_ret_5d", "GILD_Gilead_ret_20d"], "is_new": true}, {"model_id": "new_h1_GLOBAL_XGBoost_N5_t0", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 1, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "HangSeng_HK_ret_5d", "AMD_ret_5d", "DE_Deere_vol_20d"], "is_new": true}, {"model_id": "new_h1_GLOBAL_XGBoost_N5_t1", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 1, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "DOW_Price_zscore_60d", "Nikkei_Japan_vol_20d", "T_ret_1d"], "is_new": true}, {"model_id": "new_h1_GLOBAL_XGBoost_N5_t2", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 1, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "BTI_BritishAmerican_ret_20d", "EMR_Emerson_ret_20d", "ASX_Australia_vol_20d"], "is_new": true}, {"model_id": "new_h1_GLOBAL_XGBoost_N5_t3", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 1, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "PLD_Prologis_ret_5d", "ENB_EnbridgeInc_ret_1d", "SCHW_Schwab_ret_5d"], "is_new": true}, {"model_id": "new_h1_GLOBAL_XGBoost_N5_t4", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 1, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "XLV_Health_zscore_60d", "MRK_Merck_zscore_60d", "PPL_PPL_ret_1d"], "is_new": true}, {"model_id": "new_h1_GLOBAL_XGBoost_N5_t5", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 1, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "vix_acceleration_1d", "VOD_Vodafone_zscore_60d", "T10Y2Y_Spread_ret_5d"], "is_new": true}, {"model_id": "new_h1_GLOBAL_XGBoost_N5_t6", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 1, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "AMT_AmericanTower_ret_1d", "ORCL_vol_20d", "SBUX_ret_5d"], "is_new": true}, {"model_id": "new_h1_GLOBAL_XGBoost_N5_t7", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 1, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "Michigan_Sentiment_ret_20d", "BA_ret_1d", "VRP_ma5"], "is_new": true}, {"model_id": "new_h1_GLOBAL_XGBoost_N8_t0", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 1, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "US1Y_Rate_ret_5d", "EWM_Malaysia_ret_1d", "EWL_Switzerland_zscore_60d", "EWY_Korea_ret_20d", "PG_ret_20d", "EFFR_ret_1d"], "is_new": true}, {"model_id": "new_h1_GLOBAL_XGBoost_N8_t1", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 1, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "NVDA_vol_20d", "US3M_Rate_zscore_60d", "IYM_BasicMaterials_ret_20d", "HangSeng_HK_ret_1d", "CCI_CrownCastle_vol_20d", "EWS_Singapore_ret_5d"], "is_new": true}, {"model_id": "new_h1_GLOBAL_XGBoost_N8_t2", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 1, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "BDX_Becton_Dickinson_ret_20d", "ORCL_vol_20d", "Michigan_Sentiment_ret_20d", "Nikkei_Japan_zscore_60d", "EFFR_ret_1d", "NOC_Northrop_ret_20d"], "is_new": true}, {"model_id": "new_h1_GLOBAL_XGBoost_N8_t3", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 1, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "Core_PCE_zscore_60d", "HangSeng_HK_vol_20d", "EWQ_France_ret_20d", "SBUX_vol_20d", "SBUX_zscore_60d", "MS_MorganStanley_zscore_60d"], "is_new": true}, {"model_id": "new_h1_GLOBAL_XGBoost_N8_t4", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 1, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWQ_France_ret_20d", "HD_ret_5d", "EWS_Singapore_ret_5d", "XLK_Tech_zscore_60d", "ENB_EnbridgeInc_ret_1d", "Industrial_Production_zscore_60d"], "is_new": true}, {"model_id": "new_h1_GLOBAL_XGBoost_N8_t5", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 1, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "CPB_CampbellSoup_ret_5d", "CI_Cigna_vol_20d", "AMD_ret_1d", "DOW_Price_zscore_60d", "TM_Telephone_vol_20d", "Nikkei_Japan_vol_20d"], "is_new": true}, {"model_id": "new_h1_GLOBAL_XGBoost_N8_t6", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 1, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "3M_ret_5d", "PPL_PPL_ret_1d", "EQIX_Equinix_ret_5d", "ASX_Australia_ret_5d", "DE_Deere_vol_20d", "HD_zscore_60d"], "is_new": true}, {"model_id": "new_h1_GLOBAL_XGBoost_N8_t7", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 1, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "ENB_EnbridgeInc_ret_1d", "IBEX_Spain_ret_20d", "IYM_BasicMaterials_ret_20d", "CPB_CampbellSoup_ret_5d", "ASX_Australia_vol_20d", "US7Y_Rate_ret_20d"], "is_new": true}, {"model_id": "new_h1_GLOBAL_XGBoost_N10_t0", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 1, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EOG_EOGResources_ret_5d", "MS_MorganStanley_ret_5d", "SBUX_zscore_60d", "CPB_CampbellSoup_zscore_60d", "GD_GeneralDynamics_zscore_60d", "VRP_ma5", "HD_zscore_60d", "XLV_Health_zscore_60d"], "is_new": true}, {"model_id": "new_h1_GLOBAL_XGBoost_N10_t1", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 1, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "BTI_BritishAmerican_ret_20d", "PG_ret_20d", "XLF_Fin_vol_20d", "EXC_Exelon_zscore_60d", "EFFR_vol_20d", "heston_var_ev_h5", "SO_SouthernCo_ret_5d", "CPB_CampbellSoup_ret_20d"], "is_new": true}, {"model_id": "new_h1_GLOBAL_XGBoost_N10_t2", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 1, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "MS_MorganStanley_ret_5d", "Industrial_Production_zscore_60d", "vix_acceleration_1d", "EMR_Emerson_ret_20d", "QQQ_vol_20d", "PAYX_Paychex_zscore_60d", "LOW_Lowes_ret_20d", "SLB_Schlumberger_ret_5d"], "is_new": true}, {"model_id": "new_h1_GLOBAL_XGBoost_N10_t3", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 1, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWQ_France_ret_20d", "IBEX_Spain_ret_20d", "vix_acceleration_1d", "IWM_SmallCap_vol_20d", "EWH_HongKong_ret_5d", "EWQ_France_zscore_60d", "PG_ret_20d", "DOW_Price_zscore_60d"], "is_new": true}, {"model_id": "new_h1_GLOBAL_XGBoost_N10_t4", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 1, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "BLK_BlackRock_zscore_60d", "VOD_Vodafone_zscore_60d", "Core_PCE_zscore_60d", "DIS_vol_20d", "CPB_CampbellSoup_zscore_60d", "PAYX_Paychex_vol_20d", "DAX_Germany_zscore_60d", "DOW_Price_zscore_60d"], "is_new": true}, {"model_id": "new_h1_GLOBAL_XGBoost_N10_t5", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 1, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "INTC_ret_5d", "US6M_Rate_ret_20d", "EWQ_France_zscore_60d", "SBUX_ret_5d", "ORCL_vol_20d", "NEE_NextEra_ret_20d", "DAX_Germany_vol_20d", "heston_var_ev_h5"], "is_new": true}, {"model_id": "new_h1_GLOBAL_XGBoost_N10_t6", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 1, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "spx_abs_ret_max_5d", "ASX_Australia_ret_5d", "LUV_SouthwestAir_ret_5d", "EFFR_vol_20d", "MS_MorganStanley_ret_5d", "EWA_Australia_zscore_60d", "LLY_zscore_60d", "BTI_BritishAmerican_ret_20d"], "is_new": true}, {"model_id": "new_h1_GLOBAL_XGBoost_N10_t7", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 1, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "ES_Evergy_ret_1d", "SJM_JM_Smucker_ret_1d", "PG_ret_20d", "spx_momentum_3d", "MS_MorganStanley_zscore_60d", "ITT_ITTInc_ret_5d", "PAYX_Paychex_zscore_60d", "DE_Deere_vol_20d"], "is_new": true}, {"model_id": "new_h1_GLOBAL_XGBoost_N12_t0", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 1, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "3M_vol_20d", "ENB_EnbridgeInc_ret_1d", "EWY_Korea_zscore_60d", "NOC_Northrop_ret_20d", "CMCSA_ret_1d", "SBUX_vol_20d", "IYR_US_REIT2_zscore_60d", "DIS_vol_20d", "AVB_AvalonBay_zscore_60d", "DHR_ret_1d"], "is_new": true}, {"model_id": "new_h1_GLOBAL_XGBoost_N12_t1", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 1, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EQR_Equity_ret_1d", "NOC_Northrop_ret_20d", "ASX_Australia_vol_20d", "AMD_ret_1d", "gjr_condvar_h1", "LOW_Lowes_ret_5d", "JNJ_ret_1d", "XLV_Health_zscore_60d", "MRK_Merck_zscore_60d", "Industrial_Production_zscore_60d"], "is_new": true}, {"model_id": "new_h1_GLOBAL_XGBoost_N12_t2", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 1, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWJ_Japan_vol_20d", "DIS_vol_20d", "IYR_US_REIT2_zscore_60d", "spx_momentum_3d", "SLB_Schlumberger_ret_1d", "GILD_Gilead_ret_20d", "TED_Spread_zscore_60d", "MS_MorganStanley_ret_5d", "VRP_ma5", "XLK_Tech_zscore_60d"], "is_new": true}, {"model_id": "new_h1_GLOBAL_XGBoost_N12_t3", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 1, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "PCAR_PaccarInc_ret_5d", "VRP_ma5", "T_ret_1d", "HD_zscore_60d", "TM_Telephone_ret_1d", "AXP_Amex_ret_20d", "HD_ret_20d", "IYR_US_REIT2_zscore_60d", "AMGN_Amgen_ret_1d", "NEE_NextEra_ret_20d"], "is_new": true}, {"model_id": "new_h1_GLOBAL_XGBoost_N12_t4", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 1, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "PG_ret_20d", "XLF_Fin_vol_20d", "US3Y_Rate_ret_5d", "EWM_Malaysia_zscore_60d", "INTC_ret_1d", "VRP_ma5", "EWM_Malaysia_ret_1d", "PAYX_Paychex_zscore_60d", "FedFunds_zscore_60d", "HangSeng_HK_ret_1d"], "is_new": true}, {"model_id": "new_h1_GLOBAL_XGBoost_N12_t5", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 1, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "MS_MorganStanley_zscore_60d", "FedFunds_zscore_60d", "3M_vol_20d", "TED_Spread_zscore_60d", "EWM_Malaysia_vol_20d", "AVB_AvalonBay_zscore_60d", "EWL_Switzerland_zscore_60d", "PAYX_Paychex_zscore_60d", "EWQ_France_zscore_60d", "3M_ret_5d"], "is_new": true}, {"model_id": "new_h1_GLOBAL_XGBoost_N12_t6", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 1, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "TM_Telephone_ret_1d", "TXN_vol_20d", "NOC_Northrop_ret_20d", "AMGN_Amgen_ret_1d", "SJM_JM_Smucker_ret_5d", "heston_var_ev_h3", "HangSeng_HK_vol_20d", "WTI_Oil_FRED_zscore_60d", "spx_vol_5d", "BA_ret_1d"], "is_new": true}, {"model_id": "new_h1_GLOBAL_XGBoost_N12_t7", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 1, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWQ_France_ret_20d", "MS_MorganStanley_ret_1d", "US30Y_Rate_ret_20d", "IYM_BasicMaterials_ret_20d", "EQIX_Equinix_ret_5d", "TXN_vol_20d", "spx_abs_ret_max_5d", "EWG_Germany_ret_20d", "AMD_ret_5d", "Nikkei_Japan_vol_20d"], "is_new": true}, {"model_id": "new_h1_GLOBAL_XGBoost_N15_t0", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 1, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "AMT_AmericanTower_ret_1d", "CPB_CampbellSoup_ret_5d", "DE_Deere_ret_5d", "EXC_Exelon_zscore_60d", "EOG_EOGResources_ret_5d", "spx_momentum_3d", "HD_zscore_60d", "vix_mean_abs_ret_5d", "EXC_Exelon_ret_1d", "NOC_Northrop_ret_20d", "EMR_Emerson_ret_20d", "Nikkei_Japan_vol_20d", "EWG_Germany_vol_20d"], "is_new": true}, {"model_id": "new_h1_GLOBAL_XGBoost_N15_t1", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 1, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "HD_ret_5d", "IBEX_Spain_ret_20d", "TM_Telephone_vol_20d", "T_ret_1d", "heston_var_ev_h3", "BTI_BritishAmerican_ret_5d", "Brent_Oil_FRED_ret_5d", "EFFR_ret_1d", "DAX_Germany_vol_20d", "3M_ret_5d", "EWQ_France_zscore_60d", "BTI_BritishAmerican_ret_20d", "GE_ret_1d"], "is_new": true}, {"model_id": "new_h1_GLOBAL_XGBoost_N15_t2", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 1, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "DAX_Germany_vol_20d", "SJM_JM_Smucker_ret_5d", "TGT_Target_zscore_60d", "PLD_Prologis_ret_5d", "SLB_Schlumberger_ret_5d", "SBUX_ret_5d", "vix_mean_abs_ret_5d", "XLV_Health_zscore_60d", "HD_ret_1d", "CI_Cigna_vol_20d", "LOW_Lowes_ret_20d", "CMCSA_ret_1d", "LMT_LockheedMartin_vol_20d"], "is_new": true}, {"model_id": "new_h1_GLOBAL_XGBoost_N15_t3", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 1, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "AMGN_Amgen_ret_1d", "XLY_Disc_vol_20d", "CLX_Clorox_vol_20d", "BTI_BritishAmerican_ret_20d", "EMR_Emerson_ret_20d", "DIS_vol_20d", "JNJ_ret_1d", "T10Y2Y_Spread_ret_5d", "TGT_Target_zscore_60d", "EWA_Australia_zscore_60d", "Retail_Sales_zscore_60d", "gjr_condvar_h1", "LMT_LockheedMartin_ret_1d"], "is_new": true}, {"model_id": "new_h1_GLOBAL_XGBoost_N15_t4", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 1, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "CPB_CampbellSoup_vol_20d", "SO_SouthernCo_ret_5d", "Michigan_Sentiment_ret_20d", "EWJ_Japan_vol_20d", "ENB_EnbridgeInc_ret_1d", "EWA_Australia_ret_1d", "XLY_Disc_vol_20d", "heston_var_ev_h3", "SBUX_ret_5d", "VOD_Vodafone_zscore_60d", "BTI_BritishAmerican_ret_5d", "EWM_Malaysia_vol_20d", "spx_momentum_3d"], "is_new": true}, {"model_id": "new_h1_GLOBAL_XGBoost_N15_t5", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 1, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "heston_var_ev_h5", "MO_AltriaMG_ret_1d", "EWC_Canada_zscore_60d", "AVB_AvalonBay_zscore_60d", "ASX_Australia_vol_20d", "PAYX_Paychex_zscore_60d", "EWA_Australia_ret_1d", "3M_ret_5d", "US3Y_Rate_ret_5d", "LMT_LockheedMartin_vol_20d", "AMGN_Amgen_ret_1d", "vix_acceleration_1d", "MRK_Merck_zscore_60d"], "is_new": true}, {"model_id": "new_h1_GLOBAL_XGBoost_N15_t6", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 1, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "INTC_ret_1d", "PG_ret_20d", "CI_Cigna_vol_20d", "DE_Deere_vol_20d", "CPB_CampbellSoup_ret_20d", "VVIX_ret_20d", "T10Y2Y_Spread_ret_5d", "US30Y_Rate_ret_20d", "Core_CPI_zscore_60d", "VRP_ma5", "QQQ_vol_20d", "SCHW_Schwab_ret_5d", "US1Y_Rate_ret_20d"], "is_new": true}, {"model_id": "new_h1_GLOBAL_XGBoost_N15_t7", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 1, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "heston_var_ev_h3", "INTC_ret_5d", "EWS_Singapore_ret_5d", "IBEX_Spain_ret_20d", "TM_Telephone_ret_1d", "NWL_Newell_ret_20d", "HUM_Humana_ret_5d", "ASX_Australia_vol_20d", "CPB_CampbellSoup_zscore_60d", "BDX_Becton_Dickinson_ret_20d", "HangSeng_HK_ret_1d", "vix_acceleration_1d", "DHR_ret_1d"], "is_new": true}, {"model_id": "new_h1_GLOBAL_XGBoost_N20_t0", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 1, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "HD_ret_1d", "JNJ_ret_1d", "NFCI_ret_5d", "Brent_Oil_FRED_ret_5d", "TED_Spread_zscore_60d", "LUV_SouthwestAir_ret_5d", "SJM_JM_Smucker_ret_1d", "HangSeng_HK_vol_20d", "HangSeng_HK_ret_1d", "NEE_NextEra_ret_20d", "AMT_AmericanTower_ret_1d", "AMZN_ret_5d", "HD_ret_20d", "IWM_SmallCap_vol_20d", "ES_Evergy_ret_1d", "CMCSA_ret_1d", "DHR_ret_1d", "TED_Spread_vol_20d"], "is_new": true}, {"model_id": "new_h1_GLOBAL_XGBoost_N20_t1", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 1, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "CPB_CampbellSoup_ret_20d", "FedFunds_zscore_60d", "IYR_US_REIT2_zscore_60d", "INTC_ret_5d", "DAX_Germany_vol_20d", "EQR_Equity_ret_1d", "US5Y_Rate_ret_5d", "EWJ_Japan_vol_20d", "US7Y_Rate_ret_20d", "CCI_CrownCastle_vol_20d", "EWM_Malaysia_zscore_60d", "AXP_Amex_vol_20d", "GILD_Gilead_ret_20d", "IBEX_Spain_ret_20d", "T_ret_1d", "TM_Telephone_vol_20d", "TXN_vol_20d", "HangSeng_HK_ret_1d"], "is_new": true}, {"model_id": "new_h1_GLOBAL_XGBoost_N20_t2", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 1, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "AVB_AvalonBay_zscore_60d", "IBEX_Spain_ret_20d", "NVDA_vol_20d", "CMCSA_ret_1d", "spx_momentum_3d", "IYR_US_REIT2_zscore_60d", "GD_GeneralDynamics_zscore_60d", "BA_ret_1d", "DHR_ret_1d", "BTI_BritishAmerican_ret_5d", "CPB_CampbellSoup_ret_20d", "VOD_Vodafone_zscore_60d", "3M_vol_20d", "CPB_CampbellSoup_zscore_60d", "Retail_Sales_zscore_60d", "PPL_PPL_ret_1d", "LOW_Lowes_ret_5d", "Michigan_Sentiment_ret_20d"], "is_new": true}, {"model_id": "new_h1_GLOBAL_XGBoost_N20_t3", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 1, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "VRP_ma5", "US30Y_Rate_ret_20d", "EWY_Korea_ret_20d", "CCI_CrownCastle_vol_20d", "IYR_US_REIT2_zscore_60d", "EOG_EOGResources_ret_5d", "ES_Evergy_ret_1d", "hmm_p_stress", "US3M_Rate_zscore_60d", "CPB_CampbellSoup_vol_20d", "gjr_condvar_h1", "AMGN_Amgen_ret_1d", "PPL_PPL_ret_1d", "EWL_Switzerland_vol_20d", "DAX_Germany_zscore_60d", "SLB_Schlumberger_ret_5d", "EWQ_France_zscore_60d", "NEE_NextEra_ret_20d"], "is_new": true}, {"model_id": "new_h1_GLOBAL_XGBoost_N20_t4", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 1, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "T_ret_1d", "EWQ_France_ret_20d", "BLK_BlackRock_zscore_60d", "Nikkei_Japan_zscore_60d", "EWJ_Japan_vol_20d", "XLK_Tech_zscore_60d", "vix_acceleration_1d", "MS_MorganStanley_ret_1d", "EWA_Australia_zscore_60d", "TED_Spread_zscore_60d", "HD_zscore_60d", "LLY_zscore_60d", "DE_Deere_vol_20d", "Michigan_Sentiment_ret_20d", "VRP_ma5", "NFCI_ret_5d", "US6M_Rate_ret_20d", "HD_ret_5d"], "is_new": true}, {"model_id": "new_h1_GLOBAL_XGBoost_N20_t5", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 1, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EXC_Exelon_ret_1d", "heston_var_ev_h7", "XLB_Materials_zscore_60d", "SJM_JM_Smucker_ret_1d", "3M_ret_5d", "Core_CPI_zscore_60d", "Michigan_Sentiment_ret_20d", "VRP_ma5", "EWA_Australia_ret_1d", "EWY_Korea_zscore_60d", "DAX_Germany_zscore_60d", "US5Y_Rate_ret_5d", "ES_Evergy_ret_1d", "heston_var_ev_h5", "US6M_Rate_ret_20d", "Core_PCE_zscore_60d", "vix_mean_abs_ret_5d", "EWY_Korea_ret_20d"], "is_new": true}, {"model_id": "new_h1_GLOBAL_XGBoost_N20_t6", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 1, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "AXP_Amex_ret_20d", "Brent_Oil_FRED_ret_20d", "MS_MorganStanley_zscore_60d", "DE_Deere_vol_20d", "EWL_Switzerland_vol_20d", "GILD_Gilead_ret_20d", "CTAS_Cintas_vol_20d", "US5Y_Rate_ret_5d", "3M_ret_5d", "hmm_p_stress", "EWH_HongKong_ret_5d", "AMD_ret_5d", "NFCI_ret_5d", "CPB_CampbellSoup_zscore_60d", "JNJ_ret_1d", "EWG_Germany_vol_20d", "PAYX_Paychex_vol_20d", "DHR_vol_20d"], "is_new": true}, {"model_id": "new_h1_GLOBAL_XGBoost_N20_t7", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 1, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "NFCI_ret_5d", "MSTR_Bitcoin3_ret_1d", "spx_momentum_3d", "SPY_zscore_60d", "GILD_Gilead_ret_20d", "PAYX_Paychex_ret_20d", "spx_vol_5d", "EWG_Germany_ret_20d", "heston_ev_h3", "DIS_vol_20d", "EWG_Germany_vol_20d", "MS_MorganStanley_ret_1d", "LMT_LockheedMartin_vol_20d", "EWA_Australia_ret_1d", "vix_acceleration_1d", "EWS_Singapore_ret_5d", "SBUX_zscore_60d", "ORCL_vol_20d"], "is_new": true}, {"model_id": "new_h1_GLOBAL_XGBoost_N25_t0", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 1, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "PG_ret_20d", "EFFR_vol_20d", "BLK_BlackRock_zscore_60d", "spx_vol_5d", "PFE_ret_1d", "TED_Spread_vol_20d", "HD_zscore_60d", "EWG_Germany_vol_20d", "NFCI_ret_5d", "CTAS_Cintas_vol_20d", "HD_ret_5d", "DHR_vol_20d", "HD_ret_20d", "HD_ret_1d", "EQIX_Equinix_ret_5d", "Nikkei_Japan_vol_20d", "AMD_ret_5d", "QQQ_vol_20d", "TED_Spread_zscore_60d", "BTI_BritishAmerican_ret_20d", "VOD_Vodafone_zscore_60d", "PCAR_PaccarInc_ret_5d", "LMT_LockheedMartin_ret_1d"], "is_new": true}, {"model_id": "new_h1_GLOBAL_XGBoost_N25_t1", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 1, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "GILD_Gilead_ret_20d", "Retail_Sales_zscore_60d", "MS_MorganStanley_ret_1d", "GE_ret_1d", "EMR_Emerson_ret_20d", "vix_acceleration_1d", "EWL_Switzerland_zscore_60d", "ORCL_zscore_60d", "DE_Deere_vol_20d", "AMGN_Amgen_ret_1d", "EWQ_France_ret_20d", "SBUX_vol_20d", "MSTR_Bitcoin3_ret_5d", "heston_var_ev_h7", "US1Y_Rate_ret_20d", "SLB_Schlumberger_ret_5d", "DIS_vol_20d", "CTAS_Cintas_vol_20d", "GD_GeneralDynamics_zscore_60d", "MO_AltriaMG_ret_1d", "T10Y2Y_Spread_ret_5d", "EXC_Exelon_ret_1d", "CPB_CampbellSoup_ret_5d"], "is_new": true}, {"model_id": "new_h1_GLOBAL_XGBoost_N25_t2", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 1, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "BTI_BritishAmerican_ret_20d", "US3M_Rate_zscore_60d", "CPB_CampbellSoup_zscore_60d", "EWQ_France_zscore_60d", "HD_ret_20d", "EWY_Korea_zscore_60d", "EWS_Singapore_ret_5d", "US1Y_Rate_ret_5d", "XLV_Health_zscore_60d", "US5Y_Rate_ret_5d", "TED_Spread_zscore_60d", "EQR_Equity_ret_1d", "SJM_JM_Smucker_ret_1d", "gjr_condvar_h1", "3M_ret_5d", "DOW_Price_zscore_60d", "AMT_AmericanTower_ret_1d", "IWM_SmallCap_vol_20d", "US1Y_Rate_ret_20d", "NVDA_vol_20d", "vix_acceleration_1d", "JNJ_ret_1d", "heston_var_ev_h3"], "is_new": true}, {"model_id": "new_h1_GLOBAL_XGBoost_N25_t3", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 1, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "HangSeng_HK_vol_20d", "NWL_Newell_ret_20d", "EWY_Korea_zscore_60d", "GE_ret_1d", "VOD_Vodafone_zscore_60d", "EWG_Germany_ret_20d", "XOM_ret_20d", "M_Macys_vol_20d", "XLB_Materials_zscore_60d", "US3M_Rate_vol_20d", "heston_ev_h3", "LOW_Lowes_ret_20d", "CPB_CampbellSoup_ret_20d", "MSTR_Bitcoin3_ret_20d", "PPL_PPL_ret_1d", "EWC_Canada_zscore_60d", "EWS_Singapore_ret_5d", "IWM_SmallCap_vol_20d", "AXP_Amex_ret_20d", "Industrial_Production_zscore_60d", "heston_var_ev_h7", "XLV_Health_zscore_60d", "EWQ_France_zscore_60d"], "is_new": true}, {"model_id": "new_h1_GLOBAL_XGBoost_N25_t4", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 1, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "Nikkei_Japan_zscore_60d", "US30Y_Rate_ret_20d", "AMGN_Amgen_ret_1d", "Brent_Oil_FRED_ret_5d", "CPB_CampbellSoup_vol_20d", "US3M_Rate_vol_20d", "DHR_ret_1d", "EWM_Malaysia_zscore_60d", "NFCI_ret_5d", "AXP_Amex_ret_20d", "EWG_Germany_vol_20d", "PLD_Prologis_ret_5d", "heston_var_ev_h3", "TGT_Target_zscore_60d", "Brent_Oil_FRED_ret_20d", "spx_abs_ret_max_5d", "T_ret_1d", "ORCL_zscore_60d", "DE_Deere_vol_20d", "EWL_Switzerland_zscore_60d", "EWJ_Japan_vol_20d", "EWL_Switzerland_vol_20d", "MO_AltriaMG_ret_1d"], "is_new": true}, {"model_id": "new_h1_GLOBAL_XGBoost_N25_t5", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 1, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "VOD_Vodafone_zscore_60d", "NEE_NextEra_ret_20d", "Core_PCE_zscore_60d", "HD_ret_1d", "heston_var_ev_h5", "vix_mean_abs_ret_5d", "PAYX_Paychex_zscore_60d", "EWH_HongKong_ret_5d", "CMCSA_ret_1d", "JNJ_ret_1d", "EWQ_France_ret_20d", "XLK_Tech_zscore_60d", "spx_vol_5d", "heston_var_ev_h3", "EWY_Korea_zscore_60d", "WTI_Oil_FRED_zscore_60d", "TM_Telephone_ret_1d", "EWQ_France_zscore_60d", "AXP_Amex_ret_20d", "QQQ_vol_20d", "GILD_Gilead_ret_20d", "TM_Telephone_vol_20d", "ENB_EnbridgeInc_ret_1d"], "is_new": true}, {"model_id": "new_h1_GLOBAL_XGBoost_N25_t6", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 1, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "NFCI_ret_5d", "IYM_BasicMaterials_ret_20d", "CPB_CampbellSoup_ret_5d", "CLX_Clorox_vol_20d", "SJM_JM_Smucker_ret_5d", "US7Y_Rate_ret_20d", "EWM_Malaysia_vol_20d", "US5Y_Rate_ret_5d", "HD_ret_1d", "3M_ret_5d", "Nikkei_Japan_vol_20d", "LMT_LockheedMartin_ret_1d", "LOW_Lowes_ret_20d", "PAYX_Paychex_zscore_60d", "US1Y_Rate_ret_5d", "ENB_EnbridgeInc_ret_1d", "AMZN_ret_5d", "CPB_CampbellSoup_ret_20d", "US3Y_Rate_ret_5d", "AMT_AmericanTower_ret_1d", "GILD_Gilead_ret_20d", "EWA_Australia_zscore_60d", "EOG_EOGResources_vol_20d"], "is_new": true}, {"model_id": "new_h1_GLOBAL_XGBoost_N25_t7", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 1, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWC_Canada_zscore_60d", "EWH_HongKong_ret_5d", "vix_mean_abs_ret_5d", "DE_Deere_ret_5d", "HUM_Humana_ret_5d", "PFE_ret_1d", "BLK_BlackRock_zscore_60d", "PAYX_Paychex_zscore_60d", "EWM_Malaysia_ret_1d", "XLY_Disc_vol_20d", "DE_Deere_vol_20d", "ASX_Australia_ret_5d", "Core_PCE_zscore_60d", "EWG_Germany_vol_20d", "LUV_SouthwestAir_ret_5d", "HangSeng_HK_ret_1d", "Nikkei_Japan_zscore_60d", "EFFR_ret_1d", "BTI_BritishAmerican_ret_20d", "EFFR_vol_20d", "QQQ_vol_20d", "MSTR_Bitcoin3_ret_1d", "3M_vol_20d"], "is_new": true}, {"model_id": "new_h1_GLOBAL_XGBoost_N30_t0", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 1, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "NVDA_vol_20d", "CPB_CampbellSoup_zscore_60d", "CI_Cigna_vol_20d", "EFFR_vol_20d", "EWJ_Japan_vol_20d", "Brent_Oil_FRED_ret_20d", "EFFR_ret_1d", "NOC_Northrop_ret_20d", "vix_acceleration_1d", "ORCL_vol_20d", "EWM_Malaysia_vol_20d", "ENB_EnbridgeInc_ret_1d", "GD_GeneralDynamics_zscore_60d", "T10Y2Y_Spread_ret_5d", "US1Y_Rate_ret_5d", "TED_Spread_zscore_60d", "VRP_ma5", "US7Y_Rate_ret_20d", "IYM_BasicMaterials_ret_20d", "PAYX_Paychex_zscore_60d", "spx_momentum_3d", "TGT_Target_zscore_60d", "US3M_Rate_zscore_60d", "EWL_Switzerland_vol_20d", "PAYX_Paychex_vol_20d", "NWL_Newell_ret_20d", "EOG_EOGResources_vol_20d", "PAYX_Paychex_ret_20d"], "is_new": true}, {"model_id": "new_h1_GLOBAL_XGBoost_N30_t1", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 1, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "AMT_AmericanTower_ret_1d", "EWC_Canada_zscore_60d", "Michigan_Sentiment_ret_20d", "HUM_Humana_ret_5d", "MSTR_Bitcoin3_ret_1d", "TM_Telephone_ret_1d", "PLD_Prologis_ret_5d", "CMCSA_ret_1d", "US6M_Rate_ret_20d", "Core_CPI_zscore_60d", "DAX_Germany_zscore_60d", "BA_ret_1d", "EWM_Malaysia_vol_20d", "TGT_Target_zscore_60d", "CTAS_Cintas_vol_20d", "US30Y_Rate_ret_20d", "Retail_Sales_zscore_60d", "MS_MorganStanley_zscore_60d", "heston_var_ev_h3", "DE_Deere_vol_20d", "XOM_ret_20d", "SPY_zscore_60d", "EQR_Equity_ret_1d", "hmm_p_stress", "EWM_Malaysia_zscore_60d", "gjr_condvar_h1", "EWQ_France_zscore_60d", "AXP_Amex_vol_20d"], "is_new": true}, {"model_id": "new_h1_GLOBAL_XGBoost_N30_t2", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 1, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "Industrial_Production_zscore_60d", "TED_Spread_vol_20d", "TM_Telephone_ret_1d", "CMCSA_ret_1d", "EXC_Exelon_zscore_60d", "spx_abs_ret_max_5d", "IYR_US_REIT2_zscore_60d", "T_ret_1d", "DE_Deere_vol_20d", "EQIX_Equinix_ret_5d", "AXP_Amex_ret_20d", "Nikkei_Japan_vol_20d", "SBUX_zscore_60d", "EWG_Germany_vol_20d", "DHR_ret_1d", "EWL_Switzerland_zscore_60d", "CLX_Clorox_vol_20d", "SPY_zscore_60d", "DIS_vol_20d", "PAYX_Paychex_ret_20d", "Brent_Oil_FRED_ret_20d", "NWL_Newell_ret_20d", "TXN_vol_20d", "CCI_CrownCastle_vol_20d", "ITT_ITTInc_ret_5d", "PFE_ret_1d", "ENB_EnbridgeInc_ret_1d", "GILD_Gilead_ret_20d"], "is_new": true}, {"model_id": "new_h1_GLOBAL_XGBoost_N30_t3", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 1, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "AMZN_ret_5d", "vix_acceleration_1d", "vix_mean_abs_ret_5d", "AORD_AUS_zscore_60d", "CI_Cigna_vol_20d", "ITT_ITTInc_ret_5d", "ENB_EnbridgeInc_ret_1d", "heston_var_ev_h3", "US7Y_Rate_ret_20d", "CCI_CrownCastle_vol_20d", "spx_abs_ret_max_5d", "EWA_Australia_ret_1d", "SBUX_ret_5d", "CPB_CampbellSoup_ret_5d", "ORCL_zscore_60d", "TM_Telephone_ret_1d", "MRK_Merck_zscore_60d", "BDX_Becton_Dickinson_ret_20d", "spx_vol_5d", "DOW_Price_zscore_60d", "LMT_LockheedMartin_vol_20d", "INTC_ret_1d", "MSTR_Bitcoin3_ret_20d", "Core_CPI_zscore_60d", "SLB_Schlumberger_ret_1d", "HD_zscore_60d", "US3Y_Rate_ret_5d", "EXC_Exelon_zscore_60d"], "is_new": true}, {"model_id": "new_h1_GLOBAL_XGBoost_N30_t4", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 1, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "HD_ret_5d", "SBUX_zscore_60d", "T10Y2Y_Spread_ret_5d", "Brent_Oil_FRED_ret_5d", "DOW_Price_zscore_60d", "EOG_EOGResources_vol_20d", "vix_mean_abs_ret_5d", "SJM_JM_Smucker_ret_5d", "XOM_ret_1d", "AVB_AvalonBay_zscore_60d", "INTC_ret_1d", "MSTR_Bitcoin3_ret_20d", "PPL_PPL_ret_1d", "heston_ev_h3", "SLB_Schlumberger_ret_1d", "ORCL_vol_20d", "PLD_Prologis_ret_5d", "AMZN_ret_5d", "AXP_Amex_ret_20d", "AXP_Amex_vol_20d", "3M_ret_5d", "EXC_Exelon_zscore_60d", "EWS_Singapore_ret_5d", "MSTR_Bitcoin3_ret_1d", "LLY_zscore_60d", "JNJ_ret_1d", "CLX_Clorox_vol_20d", "LOW_Lowes_ret_5d"], "is_new": true}, {"model_id": "new_h1_GLOBAL_XGBoost_N30_t5", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 1, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "MSTR_Bitcoin3_ret_1d", "US3M_Rate_vol_20d", "DAX_Germany_vol_20d", "EWQ_France_ret_20d", "NFCI_ret_5d", "DOW_Price_zscore_60d", "GD_GeneralDynamics_zscore_60d", "SLB_Schlumberger_ret_5d", "EWM_Malaysia_vol_20d", "ITT_ITTInc_ret_5d", "MO_AltriaMG_ret_1d", "AXP_Amex_vol_20d", "LUV_SouthwestAir_ret_5d", "IBEX_Spain_ret_20d", "EOG_EOGResources_ret_5d", "US5Y_Rate_ret_5d", "vix_mean_abs_ret_5d", "AVB_AvalonBay_zscore_60d", "US1Y_Rate_ret_20d", "XLB_Materials_zscore_60d", "PPL_PPL_ret_1d", "BA_ret_1d", "NEE_NextEra_ret_20d", "EWH_HongKong_ret_5d", "EWQ_France_zscore_60d", "spx_abs_ret_max_5d", "US1Y_Rate_ret_5d", "PFE_ret_1d"], "is_new": true}, {"model_id": "new_h1_GLOBAL_XGBoost_N30_t6", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 1, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "DAX_Germany_zscore_60d", "BTI_BritishAmerican_ret_20d", "BLK_BlackRock_zscore_60d", "GD_GeneralDynamics_zscore_60d", "AXP_Amex_vol_20d", "DAX_Germany_vol_20d", "MS_MorganStanley_ret_5d", "EWY_Korea_zscore_60d", "TED_Spread_zscore_60d", "hmm_p_stress", "US1Y_Rate_ret_5d", "heston_var_ev_h3", "3M_vol_20d", "EXC_Exelon_zscore_60d", "XLY_Disc_vol_20d", "EQIX_Equinix_ret_5d", "T_ret_1d", "QQQ_vol_20d", "SJM_JM_Smucker_ret_1d", "EWY_Korea_ret_20d", "SPY_zscore_60d", "EWA_Australia_zscore_60d", "MSTR_Bitcoin3_ret_1d", "EWA_Australia_ret_1d", "XOM_ret_20d", "vix_acceleration_1d", "M_Macys_vol_20d", "spx_momentum_3d"], "is_new": true}, {"model_id": "new_h1_GLOBAL_XGBoost_N30_t7", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 1, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "M_Macys_vol_20d", "XLY_Disc_vol_20d", "HUM_Humana_ret_5d", "EOG_EOGResources_ret_5d", "T10Y2Y_Spread_ret_5d", "US3Y_Rate_ret_5d", "heston_var_ev_h7", "HangSeng_HK_vol_20d", "NEE_NextEra_ret_20d", "HangSeng_HK_ret_1d", "EFFR_vol_20d", "AMT_AmericanTower_ret_1d", "EWS_Singapore_ret_5d", "NVDA_vol_20d", "DE_Deere_ret_5d", "MO_AltriaMG_ret_1d", "TED_Spread_zscore_60d", "Nikkei_Japan_zscore_60d", "EWJ_Japan_vol_20d", "CPB_CampbellSoup_vol_20d", "IBEX_Spain_ret_20d", "EWY_Korea_zscore_60d", "CI_Cigna_vol_20d", "PG_ret_20d", "AVB_AvalonBay_zscore_60d", "MS_MorganStanley_ret_1d", "HD_zscore_60d", "hmm_p_stress"], "is_new": true}, {"model_id": "new_h1_GLOBAL_LightGBM_N5_t0", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 1, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "AVB_AvalonBay_zscore_60d", "ES_Evergy_ret_1d", "Michigan_Sentiment_ret_20d"], "is_new": true}, {"model_id": "new_h1_GLOBAL_LightGBM_N5_t1", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 1, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "HUM_Humana_ret_5d", "EWA_Australia_zscore_60d", "AMT_AmericanTower_ret_1d"], "is_new": true}, {"model_id": "new_h1_GLOBAL_LightGBM_N5_t2", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 1, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "HangSeng_HK_vol_20d", "NFCI_ret_5d", "HD_ret_1d"], "is_new": true}, {"model_id": "new_h1_GLOBAL_LightGBM_N5_t3", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 1, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "MO_AltriaMG_ret_1d", "EWQ_France_zscore_60d", "US1Y_Rate_ret_5d"], "is_new": true}, {"model_id": "new_h1_GLOBAL_LightGBM_N5_t4", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 1, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "DE_Deere_vol_20d", "PLD_Prologis_ret_5d", "XLK_Tech_zscore_60d"], "is_new": true}, {"model_id": "new_h1_GLOBAL_LightGBM_N5_t5", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 1, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "HD_zscore_60d", "heston_var_ev_h3", "BDX_Becton_Dickinson_ret_20d"], "is_new": true}, {"model_id": "new_h1_GLOBAL_LightGBM_N5_t6", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 1, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWH_HongKong_ret_5d", "LMT_LockheedMartin_vol_20d", "EWA_Australia_zscore_60d"], "is_new": true}, {"model_id": "new_h1_GLOBAL_LightGBM_N5_t7", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 1, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "MS_MorganStanley_zscore_60d", "EWL_Switzerland_zscore_60d", "DHR_vol_20d"], "is_new": true}, {"model_id": "new_h1_GLOBAL_LightGBM_N8_t0", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 1, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "TED_Spread_vol_20d", "PLD_Prologis_ret_5d", "DAX_Germany_vol_20d", "TXN_vol_20d", "AXP_Amex_vol_20d", "XLF_Fin_vol_20d"], "is_new": true}, {"model_id": "new_h1_GLOBAL_LightGBM_N8_t1", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 1, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "SCHW_Schwab_ret_5d", "EWL_Switzerland_vol_20d", "DE_Deere_vol_20d", "Michigan_Sentiment_ret_20d", "heston_var_ev_h5", "ITT_ITTInc_ret_5d"], "is_new": true}, {"model_id": "new_h1_GLOBAL_LightGBM_N8_t2", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 1, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "Brent_Oil_FRED_ret_5d", "CI_Cigna_vol_20d", "XLK_Tech_zscore_60d", "SLB_Schlumberger_ret_1d", "SJM_JM_Smucker_ret_1d", "IWM_SmallCap_vol_20d"], "is_new": true}, {"model_id": "new_h1_GLOBAL_LightGBM_N8_t3", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 1, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "US1Y_Rate_ret_20d", "TED_Spread_vol_20d", "AMD_ret_1d", "ES_Evergy_ret_1d", "EWH_HongKong_ret_5d", "AMZN_ret_5d"], "is_new": true}, {"model_id": "new_h1_GLOBAL_LightGBM_N8_t4", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 1, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "CTAS_Cintas_vol_20d", "T_ret_1d", "DAX_Germany_zscore_60d", "VVIX_ret_20d", "TED_Spread_vol_20d", "AMD_ret_5d"], "is_new": true}, {"model_id": "new_h1_GLOBAL_LightGBM_N8_t5", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 1, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "heston_var_ev_h3", "SJM_JM_Smucker_ret_1d", "FedFunds_zscore_60d", "HD_ret_20d", "VVIX_ret_20d", "Core_PCE_zscore_60d"], "is_new": true}, {"model_id": "new_h1_GLOBAL_LightGBM_N8_t6", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 1, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "MS_MorganStanley_ret_5d", "MO_AltriaMG_ret_1d", "XLK_Tech_zscore_60d", "LLY_zscore_60d", "DE_Deere_vol_20d", "BDX_Becton_Dickinson_ret_20d"], "is_new": true}, {"model_id": "new_h1_GLOBAL_LightGBM_N8_t7", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 1, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "TED_Spread_vol_20d", "heston_var_ev_h3", "US5Y_Rate_ret_5d", "HD_ret_5d", "CPB_CampbellSoup_zscore_60d", "TM_Telephone_vol_20d"], "is_new": true}, {"model_id": "new_h1_GLOBAL_LightGBM_N10_t0", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 1, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "TED_Spread_zscore_60d", "Retail_Sales_zscore_60d", "JNJ_ret_1d", "AMZN_ret_5d", "VOD_Vodafone_zscore_60d", "DHR_ret_1d", "NEE_NextEra_ret_20d", "EWA_Australia_zscore_60d"], "is_new": true}, {"model_id": "new_h1_GLOBAL_LightGBM_N10_t1", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 1, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "Brent_Oil_FRED_ret_20d", "EWQ_France_zscore_60d", "PAYX_Paychex_zscore_60d", "GILD_Gilead_ret_20d", "IYR_US_REIT2_zscore_60d", "Industrial_Production_zscore_60d", "NVDA_vol_20d", "Brent_Oil_FRED_ret_5d"], "is_new": true}, {"model_id": "new_h1_GLOBAL_LightGBM_N10_t2", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 1, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "DE_Deere_ret_5d", "T10Y2Y_Spread_ret_5d", "TM_Telephone_ret_1d", "NEE_NextEra_ret_20d", "AVB_AvalonBay_zscore_60d", "LUV_SouthwestAir_ret_5d", "gjr_condvar_h1", "AMT_AmericanTower_ret_1d"], "is_new": true}, {"model_id": "new_h1_GLOBAL_LightGBM_N10_t3", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 1, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "Nikkei_Japan_vol_20d", "ORCL_vol_20d", "AXP_Amex_ret_20d", "SPY_zscore_60d", "CPB_CampbellSoup_ret_20d", "Core_CPI_zscore_60d", "DAX_Germany_zscore_60d", "AMGN_Amgen_ret_1d"], "is_new": true}, {"model_id": "new_h1_GLOBAL_LightGBM_N10_t4", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 1, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "LOW_Lowes_ret_5d", "MS_MorganStanley_ret_1d", "MS_MorganStanley_zscore_60d", "XLF_Fin_vol_20d", "CLX_Clorox_vol_20d", "AMZN_ret_5d", "MSTR_Bitcoin3_ret_5d", "EFFR_ret_1d"], "is_new": true}, {"model_id": "new_h1_GLOBAL_LightGBM_N10_t5", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 1, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWJ_Japan_vol_20d", "CPB_CampbellSoup_ret_5d", "EOG_EOGResources_vol_20d", "EWL_Switzerland_vol_20d", "EXC_Exelon_ret_1d", "JNJ_ret_1d", "LOW_Lowes_ret_20d", "BA_ret_1d"], "is_new": true}, {"model_id": "new_h1_GLOBAL_LightGBM_N10_t6", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 1, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "SBUX_vol_20d", "XOM_ret_20d", "NWL_Newell_ret_20d", "US30Y_Rate_ret_20d", "heston_var_ev_h5", "DHR_vol_20d", "XOM_ret_1d", "EWH_HongKong_ret_5d"], "is_new": true}, {"model_id": "new_h1_GLOBAL_LightGBM_N10_t7", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 1, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "Core_CPI_zscore_60d", "EWQ_France_zscore_60d", "HangSeng_HK_ret_5d", "US1Y_Rate_ret_20d", "3M_vol_20d", "TED_Spread_zscore_60d", "PAYX_Paychex_vol_20d", "EWA_Australia_zscore_60d"], "is_new": true}, {"model_id": "new_h1_GLOBAL_LightGBM_N12_t0", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 1, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "vix_acceleration_1d", "T_ret_1d", "TED_Spread_zscore_60d", "DE_Deere_vol_20d", "NEE_NextEra_ret_20d", "AXP_Amex_ret_20d", "EWJ_Japan_vol_20d", "EWM_Malaysia_zscore_60d", "heston_var_ev_h7", "MRK_Merck_zscore_60d"], "is_new": true}, {"model_id": "new_h1_GLOBAL_LightGBM_N12_t1", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 1, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "vix_acceleration_1d", "MO_AltriaMG_ret_1d", "PAYX_Paychex_zscore_60d", "AMGN_Amgen_ret_1d", "XLF_Fin_vol_20d", "AXP_Amex_vol_20d", "SLB_Schlumberger_ret_1d", "US3M_Rate_vol_20d", "US6M_Rate_ret_20d", "BTI_BritishAmerican_ret_20d"], "is_new": true}, {"model_id": "new_h1_GLOBAL_LightGBM_N12_t2", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 1, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "PAYX_Paychex_zscore_60d", "EXC_Exelon_ret_1d", "US7Y_Rate_ret_20d", "ENB_EnbridgeInc_ret_1d", "Michigan_Sentiment_ret_20d", "WTI_Oil_FRED_zscore_60d", "US3Y_Rate_ret_5d", "XOM_ret_1d", "XOM_ret_20d", "NVDA_vol_20d"], "is_new": true}, {"model_id": "new_h1_GLOBAL_LightGBM_N12_t3", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 1, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "CLX_Clorox_vol_20d", "T10Y2Y_Spread_ret_5d", "HUM_Humana_ret_5d", "DIS_vol_20d", "TM_Telephone_ret_1d", "LUV_SouthwestAir_ret_5d", "US6M_Rate_ret_20d", "MSTR_Bitcoin3_ret_5d", "heston_var_ev_h3", "US3M_Rate_zscore_60d"], "is_new": true}, {"model_id": "new_h1_GLOBAL_LightGBM_N12_t4", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 1, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "HUM_Humana_ret_5d", "AMT_AmericanTower_ret_1d", "SJM_JM_Smucker_ret_5d", "IBEX_Spain_ret_20d", "US3M_Rate_zscore_60d", "XLY_Disc_vol_20d", "EWA_Australia_zscore_60d", "EOG_EOGResources_vol_20d", "EWA_Australia_ret_1d", "CPB_CampbellSoup_zscore_60d"], "is_new": true}, {"model_id": "new_h1_GLOBAL_LightGBM_N12_t5", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 1, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "Nikkei_Japan_zscore_60d", "Michigan_Sentiment_ret_20d", "US3M_Rate_vol_20d", "US30Y_Rate_ret_20d", "BDX_Becton_Dickinson_ret_20d", "NEE_NextEra_ret_20d", "XOM_ret_1d", "AMT_AmericanTower_ret_1d", "US1Y_Rate_ret_5d", "QQQ_vol_20d"], "is_new": true}, {"model_id": "new_h1_GLOBAL_LightGBM_N12_t6", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 1, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "NOC_Northrop_ret_20d", "AORD_AUS_zscore_60d", "heston_var_ev_h5", "PAYX_Paychex_ret_20d", "EOG_EOGResources_vol_20d", "EWL_Switzerland_vol_20d", "IBEX_Spain_ret_20d", "US30Y_Rate_ret_20d", "EWM_Malaysia_vol_20d", "T10Y2Y_Spread_ret_5d"], "is_new": true}, {"model_id": "new_h1_GLOBAL_LightGBM_N12_t7", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 1, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "hmm_p_stress", "EWS_Singapore_ret_5d", "T10Y2Y_Spread_ret_5d", "DHR_ret_1d", "spx_momentum_3d", "US30Y_Rate_ret_20d", "AMGN_Amgen_ret_1d", "HD_ret_20d", "3M_ret_5d", "LLY_zscore_60d"], "is_new": true}, {"model_id": "new_h1_GLOBAL_LightGBM_N15_t0", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 1, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "JNJ_ret_1d", "AORD_AUS_zscore_60d", "T_ret_1d", "DIS_vol_20d", "Michigan_Sentiment_ret_20d", "VRP_ma5", "LOW_Lowes_ret_20d", "TXN_vol_20d", "ORCL_vol_20d", "spx_vol_5d", "GD_GeneralDynamics_zscore_60d", "CMCSA_ret_1d", "MSTR_Bitcoin3_ret_20d"], "is_new": true}, {"model_id": "new_h1_GLOBAL_LightGBM_N15_t1", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 1, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "gjr_condvar_h1", "Brent_Oil_FRED_ret_20d", "EWL_Switzerland_vol_20d", "AMD_ret_5d", "US30Y_Rate_ret_20d", "SBUX_ret_5d", "Nikkei_Japan_vol_20d", "CPB_CampbellSoup_ret_5d", "DIS_vol_20d", "EWA_Australia_ret_1d", "TED_Spread_zscore_60d", "INTC_ret_1d", "US3Y_Rate_ret_5d"], "is_new": true}, {"model_id": "new_h1_GLOBAL_LightGBM_N15_t2", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 1, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "GD_GeneralDynamics_zscore_60d", "M_Macys_vol_20d", "TM_Telephone_vol_20d", "DOW_Price_zscore_60d", "EWQ_France_zscore_60d", "VOD_Vodafone_zscore_60d", "IBEX_Spain_ret_20d", "SLB_Schlumberger_ret_5d", "GE_ret_1d", "XOM_ret_20d", "AVB_AvalonBay_zscore_60d", "AORD_AUS_zscore_60d", "ENB_EnbridgeInc_ret_1d"], "is_new": true}, {"model_id": "new_h1_GLOBAL_LightGBM_N15_t3", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 1, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "vix_acceleration_1d", "SO_SouthernCo_ret_5d", "ORCL_vol_20d", "WTI_Oil_FRED_zscore_60d", "MS_MorganStanley_ret_5d", "MS_MorganStanley_ret_1d", "BTI_BritishAmerican_ret_20d", "BDX_Becton_Dickinson_ret_20d", "Core_CPI_zscore_60d", "SJM_JM_Smucker_ret_5d", "EWG_Germany_ret_20d", "Brent_Oil_FRED_ret_5d", "CCI_CrownCastle_vol_20d"], "is_new": true}, {"model_id": "new_h1_GLOBAL_LightGBM_N15_t4", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 1, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "AXP_Amex_vol_20d", "BTI_BritishAmerican_ret_20d", "MSTR_Bitcoin3_ret_1d", "TED_Spread_zscore_60d", "BDX_Becton_Dickinson_ret_20d", "XLK_Tech_zscore_60d", "heston_var_ev_h7", "HD_ret_1d", "CCI_CrownCastle_vol_20d", "HUM_Humana_ret_5d", "gjr_condvar_h1", "XLF_Fin_vol_20d", "Brent_Oil_FRED_ret_20d"], "is_new": true}, {"model_id": "new_h1_GLOBAL_LightGBM_N15_t5", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 1, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "NEE_NextEra_ret_20d", "EQIX_Equinix_ret_5d", "gjr_condvar_h1", "T10Y2Y_Spread_ret_5d", "EWC_Canada_zscore_60d", "US7Y_Rate_ret_20d", "PAYX_Paychex_ret_20d", "heston_var_ev_h5", "NFCI_ret_5d", "XOM_ret_20d", "DHR_vol_20d", "AXP_Amex_vol_20d", "MO_AltriaMG_ret_1d"], "is_new": true}, {"model_id": "new_h1_GLOBAL_LightGBM_N15_t6", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 1, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "SCHW_Schwab_ret_5d", "ORCL_zscore_60d", "PAYX_Paychex_ret_20d", "WTI_Oil_FRED_zscore_60d", "HD_ret_20d", "MS_MorganStanley_ret_5d", "EMR_Emerson_ret_20d", "BDX_Becton_Dickinson_ret_20d", "PAYX_Paychex_vol_20d", "DHR_ret_1d", "Michigan_Sentiment_ret_20d", "NFCI_ret_5d", "vix_mean_abs_ret_5d"], "is_new": true}, {"model_id": "new_h1_GLOBAL_LightGBM_N15_t7", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 1, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "US1Y_Rate_ret_5d", "CMCSA_ret_1d", "SBUX_zscore_60d", "XOM_ret_20d", "QQQ_vol_20d", "MSTR_Bitcoin3_ret_5d", "AXP_Amex_vol_20d", "EMR_Emerson_ret_20d", "US3M_Rate_vol_20d", "XLB_Materials_zscore_60d", "BTI_BritishAmerican_ret_5d", "FedFunds_zscore_60d", "AMT_AmericanTower_ret_1d"], "is_new": true}, {"model_id": "new_h1_GLOBAL_LightGBM_N20_t0", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 1, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWC_Canada_zscore_60d", "EWG_Germany_ret_20d", "PCAR_PaccarInc_ret_5d", "EWM_Malaysia_zscore_60d", "CLX_Clorox_vol_20d", "DE_Deere_ret_5d", "INTC_ret_1d", "LOW_Lowes_ret_20d", "EOG_EOGResources_ret_5d", "LLY_zscore_60d", "M_Macys_vol_20d", "CPB_CampbellSoup_zscore_60d", "PLD_Prologis_ret_5d", "TXN_vol_20d", "CCI_CrownCastle_vol_20d", "PG_ret_20d", "AMD_ret_1d", "EWG_Germany_vol_20d"], "is_new": true}, {"model_id": "new_h1_GLOBAL_LightGBM_N20_t1", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 1, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "3M_ret_5d", "MSTR_Bitcoin3_ret_20d", "CTAS_Cintas_vol_20d", "EWL_Switzerland_zscore_60d", "NWL_Newell_ret_20d", "EWQ_France_ret_20d", "SO_SouthernCo_ret_5d", "vix_mean_abs_ret_5d", "INTC_ret_5d", "DE_Deere_ret_5d", "PLD_Prologis_ret_5d", "SCHW_Schwab_ret_5d", "TXN_vol_20d", "US3Y_Rate_ret_5d", "LLY_zscore_60d", "MSTR_Bitcoin3_ret_1d", "DAX_Germany_vol_20d", "EWA_Australia_ret_1d"], "is_new": true}, {"model_id": "new_h1_GLOBAL_LightGBM_N20_t2", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 1, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "HangSeng_HK_vol_20d", "FedFunds_zscore_60d", "VRP_ma5", "AXP_Amex_vol_20d", "EOG_EOGResources_ret_5d", "US6M_Rate_ret_20d", "ITT_ITTInc_ret_5d", "DHR_vol_20d", "MS_MorganStanley_zscore_60d", "HUM_Humana_ret_5d", "SLB_Schlumberger_ret_5d", "AXP_Amex_ret_20d", "CPB_CampbellSoup_ret_20d", "Nikkei_Japan_zscore_60d", "GILD_Gilead_ret_20d", "HD_ret_20d", "BA_ret_1d", "EWQ_France_ret_20d"], "is_new": true}, {"model_id": "new_h1_GLOBAL_LightGBM_N20_t3", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 1, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "MS_MorganStanley_zscore_60d", "PPL_PPL_ret_1d", "TGT_Target_zscore_60d", "CPB_CampbellSoup_zscore_60d", "DHR_vol_20d", "PG_ret_20d", "SJM_JM_Smucker_ret_1d", "vix_acceleration_1d", "SO_SouthernCo_ret_5d", "EOG_EOGResources_ret_5d", "ES_Evergy_ret_1d", "Brent_Oil_FRED_ret_5d", "AMD_ret_5d", "EMR_Emerson_ret_20d", "heston_var_ev_h3", "CMCSA_ret_1d", "SBUX_zscore_60d", "VOD_Vodafone_zscore_60d"], "is_new": true}, {"model_id": "new_h1_GLOBAL_LightGBM_N20_t4", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 1, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EXC_Exelon_zscore_60d", "SPY_zscore_60d", "NVDA_vol_20d", "EOG_EOGResources_ret_5d", "US3Y_Rate_ret_5d", "TXN_vol_20d", "JNJ_ret_1d", "SBUX_zscore_60d", "XLF_Fin_vol_20d", "ORCL_zscore_60d", "EWM_Malaysia_zscore_60d", "Nikkei_Japan_vol_20d", "TM_Telephone_vol_20d", "EWS_Singapore_ret_5d", "Core_CPI_zscore_60d", "SO_SouthernCo_ret_5d", "EQIX_Equinix_ret_5d", "INTC_ret_5d"], "is_new": true}, {"model_id": "new_h1_GLOBAL_LightGBM_N20_t5", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 1, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "US3Y_Rate_ret_5d", "PAYX_Paychex_zscore_60d", "AMT_AmericanTower_ret_1d", "SCHW_Schwab_ret_5d", "VOD_Vodafone_zscore_60d", "TXN_vol_20d", "heston_ev_h3", "Nikkei_Japan_vol_20d", "Michigan_Sentiment_ret_20d", "Core_PCE_zscore_60d", "PCAR_PaccarInc_ret_5d", "Retail_Sales_zscore_60d", "BLK_BlackRock_zscore_60d", "SJM_JM_Smucker_ret_1d", "EWQ_France_ret_20d", "SPY_zscore_60d", "DIS_vol_20d", "AMD_ret_5d"], "is_new": true}, {"model_id": "new_h1_GLOBAL_LightGBM_N20_t6", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 1, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "CI_Cigna_vol_20d", "spx_momentum_3d", "EOG_EOGResources_ret_5d", "EQR_Equity_ret_1d", "EWM_Malaysia_ret_1d", "gjr_condvar_h1", "LMT_LockheedMartin_ret_1d", "IWM_SmallCap_vol_20d", "EFFR_vol_20d", "CLX_Clorox_vol_20d", "MS_MorganStanley_ret_5d", "LUV_SouthwestAir_ret_5d", "AMD_ret_5d", "IYR_US_REIT2_zscore_60d", "TM_Telephone_vol_20d", "PAYX_Paychex_vol_20d", "CPB_CampbellSoup_vol_20d", "FedFunds_zscore_60d"], "is_new": true}, {"model_id": "new_h1_GLOBAL_LightGBM_N20_t7", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 1, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "Brent_Oil_FRED_ret_20d", "US3Y_Rate_ret_5d", "DHR_ret_1d", "IWM_SmallCap_vol_20d", "US3M_Rate_zscore_60d", "MS_MorganStanley_zscore_60d", "PCAR_PaccarInc_ret_5d", "VVIX_ret_20d", "AVB_AvalonBay_zscore_60d", "US1Y_Rate_ret_5d", "VRP_ma5", "TGT_Target_zscore_60d", "GD_GeneralDynamics_zscore_60d", "EWQ_France_ret_20d", "Nikkei_Japan_zscore_60d", "XOM_ret_1d", "XLF_Fin_vol_20d", "TM_Telephone_vol_20d"], "is_new": true}, {"model_id": "new_h1_GLOBAL_LightGBM_N25_t0", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 1, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "ASX_Australia_vol_20d", "IYM_BasicMaterials_ret_20d", "ASX_Australia_ret_5d", "XLB_Materials_zscore_60d", "EWM_Malaysia_vol_20d", "TM_Telephone_vol_20d", "Nikkei_Japan_zscore_60d", "WTI_Oil_FRED_zscore_60d", "EWY_Korea_zscore_60d", "SLB_Schlumberger_ret_1d", "HD_zscore_60d", "DE_Deere_ret_5d", "HangSeng_HK_ret_1d", "XLF_Fin_vol_20d", "Industrial_Production_zscore_60d", "XLK_Tech_zscore_60d", "SO_SouthernCo_ret_5d", "US1Y_Rate_ret_20d", "BTI_BritishAmerican_ret_20d", "CI_Cigna_vol_20d", "HD_ret_20d", "LLY_zscore_60d", "PAYX_Paychex_zscore_60d"], "is_new": true}, {"model_id": "new_h1_GLOBAL_LightGBM_N25_t1", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 1, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "CMCSA_ret_1d", "LOW_Lowes_ret_20d", "EQIX_Equinix_ret_5d", "EWQ_France_ret_20d", "TXN_vol_20d", "M_Macys_vol_20d", "PAYX_Paychex_vol_20d", "MS_MorganStanley_ret_1d", "Retail_Sales_zscore_60d", "EWC_Canada_zscore_60d", "TM_Telephone_vol_20d", "ES_Evergy_ret_1d", "AMZN_ret_5d", "XLV_Health_zscore_60d", "PAYX_Paychex_zscore_60d", "GILD_Gilead_ret_20d", "MSTR_Bitcoin3_ret_20d", "Brent_Oil_FRED_ret_5d", "BTI_BritishAmerican_ret_20d", "PG_ret_20d", "SCHW_Schwab_ret_5d", "TGT_Target_zscore_60d", "VRP_ma5"], "is_new": true}, {"model_id": "new_h1_GLOBAL_LightGBM_N25_t2", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 1, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EFFR_ret_1d", "DE_Deere_ret_5d", "INTC_ret_5d", "SPY_zscore_60d", "US5Y_Rate_ret_5d", "EWC_Canada_zscore_60d", "SCHW_Schwab_ret_5d", "ASX_Australia_vol_20d", "TGT_Target_zscore_60d", "spx_vol_5d", "TED_Spread_vol_20d", "3M_ret_5d", "M_Macys_vol_20d", "ES_Evergy_ret_1d", "EWQ_France_ret_20d", "MO_AltriaMG_ret_1d", "BA_ret_1d", "GD_GeneralDynamics_zscore_60d", "AORD_AUS_zscore_60d", "LOW_Lowes_ret_5d", "MS_MorganStanley_ret_1d", "MSTR_Bitcoin3_ret_20d", "IYM_BasicMaterials_ret_20d"], "is_new": true}, {"model_id": "new_h1_GLOBAL_LightGBM_N25_t3", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 1, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "DAX_Germany_vol_20d", "ENB_EnbridgeInc_ret_1d", "HD_ret_1d", "TM_Telephone_vol_20d", "EQIX_Equinix_ret_5d", "JNJ_ret_1d", "MRK_Merck_zscore_60d", "GD_GeneralDynamics_zscore_60d", "EWY_Korea_zscore_60d", "EQR_Equity_ret_1d", "TED_Spread_vol_20d", "MS_MorganStanley_ret_5d", "PFE_ret_1d", "VRP_ma5", "XOM_ret_20d", "EWC_Canada_zscore_60d", "DHR_ret_1d", "LMT_LockheedMartin_vol_20d", "HUM_Humana_ret_5d", "ES_Evergy_ret_1d", "EWY_Korea_ret_20d", "DE_Deere_vol_20d", "TGT_Target_zscore_60d"], "is_new": true}, {"model_id": "new_h1_GLOBAL_LightGBM_N25_t4", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 1, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWA_Australia_zscore_60d", "TXN_vol_20d", "US3Y_Rate_ret_5d", "CPB_CampbellSoup_vol_20d", "JNJ_ret_1d", "Michigan_Sentiment_ret_20d", "ENB_EnbridgeInc_ret_1d", "EXC_Exelon_zscore_60d", "CCI_CrownCastle_vol_20d", "IWM_SmallCap_vol_20d", "ES_Evergy_ret_1d", "TGT_Target_zscore_60d", "Industrial_Production_zscore_60d", "SBUX_ret_5d", "EQIX_Equinix_ret_5d", "spx_abs_ret_max_5d", "MS_MorganStanley_ret_1d", "heston_var_ev_h7", "US5Y_Rate_ret_5d", "EOG_EOGResources_ret_5d", "hmm_p_stress", "VOD_Vodafone_zscore_60d", "EWL_Switzerland_zscore_60d"], "is_new": true}, {"model_id": "new_h1_GLOBAL_LightGBM_N25_t5", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 1, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "CPB_CampbellSoup_vol_20d", "EWM_Malaysia_ret_1d", "SLB_Schlumberger_ret_1d", "CPB_CampbellSoup_zscore_60d", "US30Y_Rate_ret_20d", "MS_MorganStanley_ret_5d", "MO_AltriaMG_ret_1d", "LUV_SouthwestAir_ret_5d", "AMZN_ret_5d", "SBUX_zscore_60d", "SPY_zscore_60d", "FedFunds_zscore_60d", "MSTR_Bitcoin3_ret_5d", "spx_abs_ret_max_5d", "IWM_SmallCap_vol_20d", "AMGN_Amgen_ret_1d", "ITT_ITTInc_ret_5d", "SBUX_vol_20d", "HD_ret_5d", "EXC_Exelon_zscore_60d", "vix_acceleration_1d", "3M_vol_20d", "EWA_Australia_zscore_60d"], "is_new": true}, {"model_id": "new_h1_GLOBAL_LightGBM_N25_t6", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 1, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "SBUX_ret_5d", "WTI_Oil_FRED_zscore_60d", "vix_mean_abs_ret_5d", "ENB_EnbridgeInc_ret_1d", "AXP_Amex_vol_20d", "EWC_Canada_zscore_60d", "T10Y2Y_Spread_ret_5d", "IYR_US_REIT2_zscore_60d", "US6M_Rate_ret_20d", "EWQ_France_ret_20d", "GD_GeneralDynamics_zscore_60d", "ASX_Australia_vol_20d", "CLX_Clorox_vol_20d", "LUV_SouthwestAir_ret_5d", "EWH_HongKong_ret_5d", "MS_MorganStanley_ret_5d", "HangSeng_HK_vol_20d", "XOM_ret_1d", "IWM_SmallCap_vol_20d", "HUM_Humana_ret_5d", "PCAR_PaccarInc_ret_5d", "CPB_CampbellSoup_zscore_60d", "vix_acceleration_1d"], "is_new": true}, {"model_id": "new_h1_GLOBAL_LightGBM_N25_t7", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 1, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EOG_EOGResources_vol_20d", "LUV_SouthwestAir_ret_5d", "XLY_Disc_vol_20d", "EWJ_Japan_vol_20d", "NVDA_vol_20d", "QQQ_vol_20d", "TED_Spread_vol_20d", "CPB_CampbellSoup_ret_20d", "EWQ_France_ret_20d", "Nikkei_Japan_vol_20d", "PAYX_Paychex_vol_20d", "INTC_ret_5d", "AXP_Amex_ret_20d", "AMGN_Amgen_ret_1d", "GD_GeneralDynamics_zscore_60d", "DOW_Price_zscore_60d", "CPB_CampbellSoup_zscore_60d", "HD_ret_20d", "CLX_Clorox_vol_20d", "ASX_Australia_vol_20d", "LMT_LockheedMartin_vol_20d", "EQIX_Equinix_ret_5d", "PCAR_PaccarInc_ret_5d"], "is_new": true}, {"model_id": "new_h1_GLOBAL_LightGBM_N30_t0", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 1, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "XLB_Materials_zscore_60d", "DE_Deere_ret_5d", "Retail_Sales_zscore_60d", "3M_vol_20d", "GE_ret_1d", "LOW_Lowes_ret_5d", "vix_acceleration_1d", "EWQ_France_ret_20d", "SO_SouthernCo_ret_5d", "TED_Spread_vol_20d", "US1Y_Rate_ret_5d", "LMT_LockheedMartin_ret_1d", "M_Macys_vol_20d", "EMR_Emerson_ret_20d", "DHR_ret_1d", "IYM_BasicMaterials_ret_20d", "gjr_condvar_h1", "Nikkei_Japan_vol_20d", "NOC_Northrop_ret_20d", "BDX_Becton_Dickinson_ret_20d", "IWM_SmallCap_vol_20d", "US7Y_Rate_ret_20d", "SLB_Schlumberger_ret_1d", "WTI_Oil_FRED_zscore_60d", "PAYX_Paychex_zscore_60d", "TM_Telephone_ret_1d", "PAYX_Paychex_ret_20d", "HD_ret_1d"], "is_new": true}, {"model_id": "new_h1_GLOBAL_LightGBM_N30_t1", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 1, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "Industrial_Production_zscore_60d", "SLB_Schlumberger_ret_5d", "vix_mean_abs_ret_5d", "HD_ret_1d", "TM_Telephone_vol_20d", "gjr_condvar_h1", "spx_vol_5d", "VOD_Vodafone_zscore_60d", "NWL_Newell_ret_20d", "LUV_SouthwestAir_ret_5d", "INTC_ret_5d", "XLV_Health_zscore_60d", "HD_ret_5d", "EQIX_Equinix_ret_5d", "SJM_JM_Smucker_ret_5d", "DAX_Germany_zscore_60d", "HangSeng_HK_ret_5d", "Core_PCE_zscore_60d", "CI_Cigna_vol_20d", "Nikkei_Japan_vol_20d", "IYR_US_REIT2_zscore_60d", "AMZN_ret_5d", "DE_Deere_vol_20d", "3M_vol_20d", "AVB_AvalonBay_zscore_60d", "US5Y_Rate_ret_5d", "EFFR_ret_1d", "HangSeng_HK_vol_20d"], "is_new": true}, {"model_id": "new_h1_GLOBAL_LightGBM_N30_t2", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 1, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWQ_France_ret_20d", "DE_Deere_ret_5d", "heston_var_ev_h5", "Core_CPI_zscore_60d", "IWM_SmallCap_vol_20d", "INTC_ret_5d", "BTI_BritishAmerican_ret_5d", "EWH_HongKong_ret_5d", "3M_vol_20d", "ITT_ITTInc_ret_5d", "JNJ_ret_1d", "SBUX_ret_5d", "ASX_Australia_ret_5d", "T10Y2Y_Spread_ret_5d", "EWG_Germany_vol_20d", "NWL_Newell_ret_20d", "US1Y_Rate_ret_5d", "EQR_Equity_ret_1d", "CPB_CampbellSoup_ret_5d", "HD_ret_20d", "HangSeng_HK_ret_5d", "VVIX_ret_20d", "XLY_Disc_vol_20d", "TED_Spread_vol_20d", "MSTR_Bitcoin3_ret_5d", "US5Y_Rate_ret_5d", "SBUX_zscore_60d", "DHR_vol_20d"], "is_new": true}, {"model_id": "new_h1_GLOBAL_LightGBM_N30_t3", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 1, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "heston_var_ev_h3", "ORCL_zscore_60d", "HD_zscore_60d", "heston_var_ev_h5", "NFCI_ret_5d", "DOW_Price_zscore_60d", "3M_vol_20d", "DE_Deere_ret_5d", "LUV_SouthwestAir_ret_5d", "SO_SouthernCo_ret_5d", "ENB_EnbridgeInc_ret_1d", "MO_AltriaMG_ret_1d", "HD_ret_5d", "EWH_HongKong_ret_5d", "BTI_BritishAmerican_ret_5d", "TED_Spread_vol_20d", "EWL_Switzerland_vol_20d", "EOG_EOGResources_vol_20d", "TXN_vol_20d", "CPB_CampbellSoup_zscore_60d", "DE_Deere_vol_20d", "EWA_Australia_ret_1d", "BTI_BritishAmerican_ret_20d", "DAX_Germany_zscore_60d", "EWG_Germany_ret_20d", "DHR_vol_20d", "heston_var_ev_h7", "HangSeng_HK_vol_20d"], "is_new": true}, {"model_id": "new_h1_GLOBAL_LightGBM_N30_t4", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 1, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "AMD_ret_1d", "CMCSA_ret_1d", "T10Y2Y_Spread_ret_5d", "EWS_Singapore_ret_5d", "NOC_Northrop_ret_20d", "FedFunds_zscore_60d", "SBUX_zscore_60d", "EFFR_vol_20d", "EWG_Germany_vol_20d", "US1Y_Rate_ret_20d", "PG_ret_20d", "LLY_zscore_60d", "AMZN_ret_5d", "TGT_Target_zscore_60d", "ENB_EnbridgeInc_ret_1d", "PPL_PPL_ret_1d", "Michigan_Sentiment_ret_20d", "SPY_zscore_60d", "CLX_Clorox_vol_20d", "PCAR_PaccarInc_ret_5d", "EQIX_Equinix_ret_5d", "EQR_Equity_ret_1d", "LMT_LockheedMartin_ret_1d", "Retail_Sales_zscore_60d", "gjr_condvar_h1", "ITT_ITTInc_ret_5d", "EXC_Exelon_zscore_60d", "LOW_Lowes_ret_5d"], "is_new": true}, {"model_id": "new_h1_GLOBAL_LightGBM_N30_t5", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 1, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "MSTR_Bitcoin3_ret_20d", "EFFR_ret_1d", "US3Y_Rate_ret_5d", "ASX_Australia_ret_5d", "IWM_SmallCap_vol_20d", "DHR_vol_20d", "XLY_Disc_vol_20d", "CMCSA_ret_1d", "SLB_Schlumberger_ret_5d", "PCAR_PaccarInc_ret_5d", "vix_mean_abs_ret_5d", "US1Y_Rate_ret_20d", "EWQ_France_zscore_60d", "CPB_CampbellSoup_vol_20d", "MS_MorganStanley_ret_5d", "BA_ret_1d", "T_ret_1d", "EQR_Equity_ret_1d", "ORCL_zscore_60d", "Core_PCE_zscore_60d", "PPL_PPL_ret_1d", "EOG_EOGResources_vol_20d", "BTI_BritishAmerican_ret_20d", "PFE_ret_1d", "EWL_Switzerland_vol_20d", "EWC_Canada_zscore_60d", "PAYX_Paychex_ret_20d", "US5Y_Rate_ret_5d"], "is_new": true}, {"model_id": "new_h1_GLOBAL_LightGBM_N30_t6", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 1, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EQR_Equity_ret_1d", "EOG_EOGResources_vol_20d", "EFFR_ret_1d", "US5Y_Rate_ret_5d", "US3M_Rate_zscore_60d", "XLV_Health_zscore_60d", "EWJ_Japan_vol_20d", "NFCI_ret_5d", "EOG_EOGResources_ret_5d", "CPB_CampbellSoup_ret_5d", "Nikkei_Japan_vol_20d", "IWM_SmallCap_vol_20d", "US7Y_Rate_ret_20d", "Brent_Oil_FRED_ret_5d", "HangSeng_HK_ret_5d", "EWQ_France_ret_20d", "US1Y_Rate_ret_20d", "T_ret_1d", "DHR_vol_20d", "XLK_Tech_zscore_60d", "ENB_EnbridgeInc_ret_1d", "XOM_ret_20d", "LUV_SouthwestAir_ret_5d", "EWS_Singapore_ret_5d", "spx_momentum_3d", "TM_Telephone_vol_20d", "EWG_Germany_vol_20d", "INTC_ret_5d"], "is_new": true}, {"model_id": "new_h1_GLOBAL_LightGBM_N30_t7", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 1, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "DAX_Germany_vol_20d", "SBUX_zscore_60d", "heston_var_ev_h7", "PG_ret_20d", "SCHW_Schwab_ret_5d", "EWL_Switzerland_zscore_60d", "EMR_Emerson_ret_20d", "MS_MorganStanley_ret_5d", "HD_zscore_60d", "EWY_Korea_ret_20d", "AMT_AmericanTower_ret_1d", "US3Y_Rate_ret_5d", "IYR_US_REIT2_zscore_60d", "EXC_Exelon_zscore_60d", "EWY_Korea_zscore_60d", "US5Y_Rate_ret_5d", "SJM_JM_Smucker_ret_1d", "Core_CPI_zscore_60d", "EQR_Equity_ret_1d", "PAYX_Paychex_vol_20d", "US7Y_Rate_ret_20d", "ORCL_zscore_60d", "3M_ret_5d", "ES_Evergy_ret_1d", "XOM_ret_20d", "EOG_EOGResources_vol_20d", "VOD_Vodafone_zscore_60d", "BTI_BritishAmerican_ret_5d"], "is_new": true}, {"model_id": "new_h1_GLOBAL_GradientBoosting_N5_t0", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 1, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "vix_acceleration_1d", "VVIX_ret_20d", "PLD_Prologis_ret_5d"], "is_new": true}, {"model_id": "new_h1_GLOBAL_GradientBoosting_N5_t1", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 1, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "SBUX_zscore_60d", "IWM_SmallCap_vol_20d", "DHR_ret_1d"], "is_new": true}, {"model_id": "new_h1_GLOBAL_GradientBoosting_N5_t2", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 1, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "AORD_AUS_zscore_60d", "CPB_CampbellSoup_ret_5d", "DE_Deere_vol_20d"], "is_new": true}, {"model_id": "new_h1_GLOBAL_GradientBoosting_N5_t3", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 1, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EOG_EOGResources_vol_20d", "ASX_Australia_vol_20d", "AXP_Amex_ret_20d"], "is_new": true}, {"model_id": "new_h1_GLOBAL_GradientBoosting_N5_t4", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 1, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "LMT_LockheedMartin_ret_1d", "EWH_HongKong_ret_5d", "MRK_Merck_zscore_60d"], "is_new": true}, {"model_id": "new_h1_GLOBAL_GradientBoosting_N5_t5", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 1, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "heston_var_ev_h3", "TGT_Target_zscore_60d", "MS_MorganStanley_ret_5d"], "is_new": true}, {"model_id": "new_h1_GLOBAL_GradientBoosting_N5_t6", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 1, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "HD_ret_1d", "HangSeng_HK_ret_5d", "EWQ_France_ret_20d"], "is_new": true}, {"model_id": "new_h1_GLOBAL_GradientBoosting_N5_t7", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 1, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "PFE_ret_1d", "BTI_BritishAmerican_ret_5d", "LLY_zscore_60d"], "is_new": true}, {"model_id": "new_h1_GLOBAL_GradientBoosting_N8_t0", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 1, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "DOW_Price_zscore_60d", "heston_var_ev_h5", "US5Y_Rate_ret_5d", "ENB_EnbridgeInc_ret_1d", "VOD_Vodafone_zscore_60d", "LMT_LockheedMartin_ret_1d"], "is_new": true}, {"model_id": "new_h1_GLOBAL_GradientBoosting_N8_t1", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 1, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWC_Canada_zscore_60d", "Retail_Sales_zscore_60d", "GILD_Gilead_ret_20d", "spx_abs_ret_max_5d", "SLB_Schlumberger_ret_5d", "EWA_Australia_zscore_60d"], "is_new": true}, {"model_id": "new_h1_GLOBAL_GradientBoosting_N8_t2", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 1, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "PPL_PPL_ret_1d", "CI_Cigna_vol_20d", "EWH_HongKong_ret_5d", "VRP_ma5", "SJM_JM_Smucker_ret_1d", "CMCSA_ret_1d"], "is_new": true}, {"model_id": "new_h1_GLOBAL_GradientBoosting_N8_t3", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 1, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "AXP_Amex_vol_20d", "heston_ev_h3", "DOW_Price_zscore_60d", "Core_CPI_zscore_60d", "EWH_HongKong_ret_5d", "EFFR_vol_20d"], "is_new": true}, {"model_id": "new_h1_GLOBAL_GradientBoosting_N8_t4", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 1, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "spx_vol_5d", "DHR_ret_1d", "BA_ret_1d", "Core_CPI_zscore_60d", "XLF_Fin_vol_20d", "CLX_Clorox_vol_20d"], "is_new": true}, {"model_id": "new_h1_GLOBAL_GradientBoosting_N8_t5", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 1, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "CPB_CampbellSoup_vol_20d", "DE_Deere_ret_5d", "EQR_Equity_ret_1d", "AVB_AvalonBay_zscore_60d", "Core_PCE_zscore_60d", "XLY_Disc_vol_20d"], "is_new": true}, {"model_id": "new_h1_GLOBAL_GradientBoosting_N8_t6", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 1, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "JNJ_ret_1d", "spx_vol_5d", "TED_Spread_zscore_60d", "ORCL_zscore_60d", "EWY_Korea_ret_20d", "EWC_Canada_zscore_60d"], "is_new": true}, {"model_id": "new_h1_GLOBAL_GradientBoosting_N8_t7", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 1, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "US3Y_Rate_ret_5d", "EWA_Australia_zscore_60d", "Nikkei_Japan_zscore_60d", "MSTR_Bitcoin3_ret_20d", "Industrial_Production_zscore_60d", "EWH_HongKong_ret_5d"], "is_new": true}, {"model_id": "new_h1_GLOBAL_GradientBoosting_N10_t0", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 1, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "hmm_p_stress", "vix_acceleration_1d", "HangSeng_HK_ret_5d", "T10Y2Y_Spread_ret_5d", "CPB_CampbellSoup_vol_20d", "DOW_Price_zscore_60d", "HD_ret_20d", "EQIX_Equinix_ret_5d"], "is_new": true}, {"model_id": "new_h1_GLOBAL_GradientBoosting_N10_t1", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 1, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "US1Y_Rate_ret_5d", "MS_MorganStanley_zscore_60d", "XOM_ret_20d", "TED_Spread_vol_20d", "DHR_ret_1d", "Industrial_Production_zscore_60d", "AMD_ret_1d", "US5Y_Rate_ret_5d"], "is_new": true}, {"model_id": "new_h1_GLOBAL_GradientBoosting_N10_t2", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 1, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "VVIX_ret_20d", "HD_ret_5d", "MSTR_Bitcoin3_ret_1d", "DAX_Germany_zscore_60d", "AMGN_Amgen_ret_1d", "VRP_ma5", "EWA_Australia_zscore_60d", "LUV_SouthwestAir_ret_5d"], "is_new": true}, {"model_id": "new_h1_GLOBAL_GradientBoosting_N10_t3", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 1, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWJ_Japan_vol_20d", "vix_acceleration_1d", "US3M_Rate_zscore_60d", "VVIX_ret_20d", "LLY_zscore_60d", "DAX_Germany_zscore_60d", "US7Y_Rate_ret_20d", "SJM_JM_Smucker_ret_1d"], "is_new": true}, {"model_id": "new_h1_GLOBAL_GradientBoosting_N10_t4", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 1, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "Core_PCE_zscore_60d", "DHR_vol_20d", "GILD_Gilead_ret_20d", "EWA_Australia_ret_1d", "NWL_Newell_ret_20d", "LOW_Lowes_ret_20d", "T_ret_1d", "IWM_SmallCap_vol_20d"], "is_new": true}, {"model_id": "new_h1_GLOBAL_GradientBoosting_N10_t5", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 1, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "DE_Deere_ret_5d", "EWJ_Japan_vol_20d", "AMZN_ret_5d", "EXC_Exelon_zscore_60d", "Brent_Oil_FRED_ret_5d", "EFFR_ret_1d", "DHR_vol_20d", "SBUX_ret_5d"], "is_new": true}, {"model_id": "new_h1_GLOBAL_GradientBoosting_N10_t6", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 1, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "XLV_Health_zscore_60d", "EMR_Emerson_ret_20d", "Retail_Sales_zscore_60d", "EWM_Malaysia_zscore_60d", "US1Y_Rate_ret_5d", "SCHW_Schwab_ret_5d", "HD_ret_1d", "BLK_BlackRock_zscore_60d"], "is_new": true}, {"model_id": "new_h1_GLOBAL_GradientBoosting_N10_t7", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 1, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "Nikkei_Japan_zscore_60d", "Nikkei_Japan_vol_20d", "VRP_ma5", "AMT_AmericanTower_ret_1d", "NFCI_ret_5d", "EWM_Malaysia_vol_20d", "DE_Deere_ret_5d", "DAX_Germany_zscore_60d"], "is_new": true}, {"model_id": "new_h1_GLOBAL_GradientBoosting_N12_t0", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 1, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EXC_Exelon_zscore_60d", "EFFR_vol_20d", "gjr_condvar_h1", "heston_var_ev_h5", "3M_vol_20d", "EWL_Switzerland_zscore_60d", "HD_ret_5d", "ASX_Australia_ret_5d", "EWY_Korea_ret_20d", "BLK_BlackRock_zscore_60d"], "is_new": true}, {"model_id": "new_h1_GLOBAL_GradientBoosting_N12_t1", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 1, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "SJM_JM_Smucker_ret_1d", "US3M_Rate_vol_20d", "heston_var_ev_h3", "PAYX_Paychex_zscore_60d", "MO_AltriaMG_ret_1d", "NWL_Newell_ret_20d", "HangSeng_HK_vol_20d", "XOM_ret_1d", "BTI_BritishAmerican_ret_20d", "PAYX_Paychex_ret_20d"], "is_new": true}, {"model_id": "new_h1_GLOBAL_GradientBoosting_N12_t2", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 1, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "NWL_Newell_ret_20d", "EOG_EOGResources_vol_20d", "EWY_Korea_ret_20d", "EWY_Korea_zscore_60d", "Retail_Sales_zscore_60d", "CPB_CampbellSoup_zscore_60d", "DHR_ret_1d", "PG_ret_20d", "ORCL_vol_20d", "3M_ret_5d"], "is_new": true}, {"model_id": "new_h1_GLOBAL_GradientBoosting_N12_t3", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 1, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "SO_SouthernCo_ret_5d", "DAX_Germany_vol_20d", "ASX_Australia_vol_20d", "EWH_HongKong_ret_5d", "AMZN_ret_5d", "NFCI_ret_5d", "US1Y_Rate_ret_20d", "EWM_Malaysia_vol_20d", "CPB_CampbellSoup_ret_5d", "PAYX_Paychex_zscore_60d"], "is_new": true}, {"model_id": "new_h1_GLOBAL_GradientBoosting_N12_t4", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 1, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EMR_Emerson_ret_20d", "NWL_Newell_ret_20d", "SPY_zscore_60d", "AVB_AvalonBay_zscore_60d", "TXN_vol_20d", "AMD_ret_5d", "CPB_CampbellSoup_ret_20d", "vix_mean_abs_ret_5d", "LUV_SouthwestAir_ret_5d", "PAYX_Paychex_vol_20d"], "is_new": true}, {"model_id": "new_h1_GLOBAL_GradientBoosting_N12_t5", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 1, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "JNJ_ret_1d", "BLK_BlackRock_zscore_60d", "PG_ret_20d", "CI_Cigna_vol_20d", "EWC_Canada_zscore_60d", "SLB_Schlumberger_ret_5d", "AVB_AvalonBay_zscore_60d", "NEE_NextEra_ret_20d", "PAYX_Paychex_zscore_60d", "CPB_CampbellSoup_vol_20d"], "is_new": true}, {"model_id": "new_h1_GLOBAL_GradientBoosting_N12_t6", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 1, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "IYR_US_REIT2_zscore_60d", "CCI_CrownCastle_vol_20d", "AXP_Amex_vol_20d", "HD_ret_5d", "LOW_Lowes_ret_5d", "TED_Spread_zscore_60d", "GILD_Gilead_ret_20d", "SBUX_zscore_60d", "HangSeng_HK_ret_1d", "EWG_Germany_ret_20d"], "is_new": true}, {"model_id": "new_h1_GLOBAL_GradientBoosting_N12_t7", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 1, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "AMZN_ret_5d", "AMGN_Amgen_ret_1d", "US3M_Rate_vol_20d", "ORCL_vol_20d", "EQR_Equity_ret_1d", "LMT_LockheedMartin_ret_1d", "TGT_Target_zscore_60d", "TED_Spread_zscore_60d", "LMT_LockheedMartin_vol_20d", "PAYX_Paychex_vol_20d"], "is_new": true}, {"model_id": "new_h1_GLOBAL_GradientBoosting_N15_t0", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 1, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "AORD_AUS_zscore_60d", "EWG_Germany_vol_20d", "Core_PCE_zscore_60d", "US30Y_Rate_ret_20d", "3M_vol_20d", "IYR_US_REIT2_zscore_60d", "EWC_Canada_zscore_60d", "vix_mean_abs_ret_5d", "EWG_Germany_ret_20d", "XLF_Fin_vol_20d", "Michigan_Sentiment_ret_20d", "EFFR_vol_20d", "ORCL_zscore_60d"], "is_new": true}, {"model_id": "new_h1_GLOBAL_GradientBoosting_N15_t1", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 1, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "DOW_Price_zscore_60d", "CCI_CrownCastle_vol_20d", "LMT_LockheedMartin_vol_20d", "US7Y_Rate_ret_20d", "BTI_BritishAmerican_ret_20d", "TED_Spread_vol_20d", "EWQ_France_zscore_60d", "DHR_ret_1d", "QQQ_vol_20d", "AMGN_Amgen_ret_1d", "ASX_Australia_ret_5d", "BDX_Becton_Dickinson_ret_20d", "GD_GeneralDynamics_zscore_60d"], "is_new": true}, {"model_id": "new_h1_GLOBAL_GradientBoosting_N15_t2", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 1, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "IBEX_Spain_ret_20d", "IYM_BasicMaterials_ret_20d", "XOM_ret_20d", "VOD_Vodafone_zscore_60d", "AMZN_ret_5d", "BTI_BritishAmerican_ret_5d", "Core_CPI_zscore_60d", "AXP_Amex_vol_20d", "PAYX_Paychex_vol_20d", "SBUX_vol_20d", "vix_acceleration_1d", "PLD_Prologis_ret_5d", "DAX_Germany_vol_20d"], "is_new": true}, {"model_id": "new_h1_GLOBAL_GradientBoosting_N15_t3", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 1, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "US7Y_Rate_ret_20d", "EWQ_France_ret_20d", "TXN_vol_20d", "US1Y_Rate_ret_20d", "BTI_BritishAmerican_ret_5d", "US3M_Rate_vol_20d", "gjr_condvar_h1", "GE_ret_1d", "US3Y_Rate_ret_5d", "3M_ret_5d", "INTC_ret_1d", "EWY_Korea_ret_20d", "vix_acceleration_1d"], "is_new": true}, {"model_id": "new_h1_GLOBAL_GradientBoosting_N15_t4", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 1, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "AVB_AvalonBay_zscore_60d", "EXC_Exelon_ret_1d", "XLK_Tech_zscore_60d", "XLF_Fin_vol_20d", "SJM_JM_Smucker_ret_5d", "ASX_Australia_vol_20d", "VOD_Vodafone_zscore_60d", "HangSeng_HK_ret_1d", "heston_var_ev_h7", "WTI_Oil_FRED_zscore_60d", "SJM_JM_Smucker_ret_1d", "MS_MorganStanley_zscore_60d", "TED_Spread_zscore_60d"], "is_new": true}, {"model_id": "new_h1_GLOBAL_GradientBoosting_N15_t5", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 1, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "PG_ret_20d", "VRP_ma5", "ORCL_zscore_60d", "Core_PCE_zscore_60d", "NOC_Northrop_ret_20d", "MSTR_Bitcoin3_ret_20d", "XLB_Materials_zscore_60d", "US1Y_Rate_ret_5d", "HD_zscore_60d", "CPB_CampbellSoup_vol_20d", "NFCI_ret_5d", "CPB_CampbellSoup_ret_5d", "DOW_Price_zscore_60d"], "is_new": true}, {"model_id": "new_h1_GLOBAL_GradientBoosting_N15_t6", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 1, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "SBUX_vol_20d", "Retail_Sales_zscore_60d", "US3Y_Rate_ret_5d", "GD_GeneralDynamics_zscore_60d", "EWY_Korea_ret_20d", "EWM_Malaysia_zscore_60d", "EOG_EOGResources_vol_20d", "ITT_ITTInc_ret_5d", "ORCL_vol_20d", "EQR_Equity_ret_1d", "EWH_HongKong_ret_5d", "SO_SouthernCo_ret_5d", "vix_mean_abs_ret_5d"], "is_new": true}, {"model_id": "new_h1_GLOBAL_GradientBoosting_N15_t7", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 1, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "AMD_ret_5d", "BTI_BritishAmerican_ret_5d", "FedFunds_zscore_60d", "ENB_EnbridgeInc_ret_1d", "Nikkei_Japan_zscore_60d", "spx_vol_5d", "DHR_ret_1d", "AMZN_ret_5d", "EWJ_Japan_vol_20d", "PLD_Prologis_ret_5d", "BA_ret_1d", "SO_SouthernCo_ret_5d", "PFE_ret_1d"], "is_new": true}, {"model_id": "new_h1_GLOBAL_GradientBoosting_N20_t0", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 1, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "NFCI_ret_5d", "BDX_Becton_Dickinson_ret_20d", "EFFR_vol_20d", "spx_vol_5d", "vix_acceleration_1d", "spx_momentum_3d", "AORD_AUS_zscore_60d", "gjr_condvar_h1", "XOM_ret_20d", "EXC_Exelon_zscore_60d", "CPB_CampbellSoup_ret_20d", "TXN_vol_20d", "CI_Cigna_vol_20d", "EWQ_France_ret_20d", "MO_AltriaMG_ret_1d", "XLV_Health_zscore_60d", "US3Y_Rate_ret_5d", "SO_SouthernCo_ret_5d"], "is_new": true}, {"model_id": "new_h1_GLOBAL_GradientBoosting_N20_t1", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 1, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "CI_Cigna_vol_20d", "PAYX_Paychex_ret_20d", "AMT_AmericanTower_ret_1d", "TM_Telephone_ret_1d", "SJM_JM_Smucker_ret_5d", "XOM_ret_20d", "VOD_Vodafone_zscore_60d", "HD_zscore_60d", "spx_vol_5d", "SLB_Schlumberger_ret_5d", "AORD_AUS_zscore_60d", "Brent_Oil_FRED_ret_20d", "TXN_vol_20d", "M_Macys_vol_20d", "EXC_Exelon_zscore_60d", "HangSeng_HK_ret_5d", "TGT_Target_zscore_60d", "DHR_ret_1d"], "is_new": true}, {"model_id": "new_h1_GLOBAL_GradientBoosting_N20_t2", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 1, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "T10Y2Y_Spread_ret_5d", "SO_SouthernCo_ret_5d", "EWM_Malaysia_zscore_60d", "US3Y_Rate_ret_5d", "ORCL_zscore_60d", "SLB_Schlumberger_ret_1d", "MRK_Merck_zscore_60d", "AMD_ret_1d", "TM_Telephone_vol_20d", "EOG_EOGResources_vol_20d", "BLK_BlackRock_zscore_60d", "AMZN_ret_5d", "EXC_Exelon_zscore_60d", "AXP_Amex_ret_20d", "NVDA_vol_20d", "QQQ_vol_20d", "INTC_ret_5d", "vix_acceleration_1d"], "is_new": true}, {"model_id": "new_h1_GLOBAL_GradientBoosting_N20_t3", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 1, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "US3M_Rate_vol_20d", "CI_Cigna_vol_20d", "HangSeng_HK_ret_1d", "VVIX_ret_20d", "Michigan_Sentiment_ret_20d", "CMCSA_ret_1d", "heston_var_ev_h7", "vix_mean_abs_ret_5d", "EFFR_vol_20d", "SLB_Schlumberger_ret_5d", "SJM_JM_Smucker_ret_1d", "VRP_ma5", "GE_ret_1d", "ENB_EnbridgeInc_ret_1d", "WTI_Oil_FRED_zscore_60d", "NWL_Newell_ret_20d", "US30Y_Rate_ret_20d", "SLB_Schlumberger_ret_1d"], "is_new": true}, {"model_id": "new_h1_GLOBAL_GradientBoosting_N20_t4", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 1, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "vix_acceleration_1d", "heston_ev_h3", "AXP_Amex_vol_20d", "GD_GeneralDynamics_zscore_60d", "AXP_Amex_ret_20d", "VOD_Vodafone_zscore_60d", "LMT_LockheedMartin_vol_20d", "EWM_Malaysia_vol_20d", "Core_PCE_zscore_60d", "EXC_Exelon_zscore_60d", "CLX_Clorox_vol_20d", "US6M_Rate_ret_20d", "NWL_Newell_ret_20d", "CTAS_Cintas_vol_20d", "Nikkei_Japan_vol_20d", "SCHW_Schwab_ret_5d", "US1Y_Rate_ret_20d", "heston_var_ev_h5"], "is_new": true}, {"model_id": "new_h1_GLOBAL_GradientBoosting_N20_t5", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 1, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "US1Y_Rate_ret_20d", "US1Y_Rate_ret_5d", "EWM_Malaysia_ret_1d", "IYR_US_REIT2_zscore_60d", "MSTR_Bitcoin3_ret_1d", "LMT_LockheedMartin_vol_20d", "CPB_CampbellSoup_zscore_60d", "PFE_ret_1d", "US6M_Rate_ret_20d", "spx_momentum_3d", "vix_mean_abs_ret_5d", "heston_var_ev_h5", "MS_MorganStanley_zscore_60d", "EQR_Equity_ret_1d", "HD_zscore_60d", "heston_var_ev_h3", "AXP_Amex_ret_20d", "XLB_Materials_zscore_60d"], "is_new": true}, {"model_id": "new_h1_GLOBAL_GradientBoosting_N20_t6", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 1, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "TXN_vol_20d", "EWY_Korea_ret_20d", "FedFunds_zscore_60d", "XLK_Tech_zscore_60d", "Nikkei_Japan_vol_20d", "AMZN_ret_5d", "GD_GeneralDynamics_zscore_60d", "PAYX_Paychex_zscore_60d", "SLB_Schlumberger_ret_5d", "IBEX_Spain_ret_20d", "DOW_Price_zscore_60d", "BTI_BritishAmerican_ret_20d", "PAYX_Paychex_ret_20d", "AMD_ret_1d", "EFFR_ret_1d", "US30Y_Rate_ret_20d", "Industrial_Production_zscore_60d", "CPB_CampbellSoup_vol_20d"], "is_new": true}, {"model_id": "new_h1_GLOBAL_GradientBoosting_N20_t7", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 1, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "US6M_Rate_ret_20d", "EWS_Singapore_ret_5d", "gjr_condvar_h1", "NEE_NextEra_ret_20d", "GILD_Gilead_ret_20d", "TM_Telephone_vol_20d", "TXN_vol_20d", "DAX_Germany_zscore_60d", "3M_ret_5d", "SJM_JM_Smucker_ret_5d", "US3Y_Rate_ret_5d", "EWL_Switzerland_zscore_60d", "GD_GeneralDynamics_zscore_60d", "TGT_Target_zscore_60d", "M_Macys_vol_20d", "EWM_Malaysia_vol_20d", "XLV_Health_zscore_60d", "AVB_AvalonBay_zscore_60d"], "is_new": true}, {"model_id": "new_h1_GLOBAL_GradientBoosting_N25_t0", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 1, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "Retail_Sales_zscore_60d", "DOW_Price_zscore_60d", "TED_Spread_vol_20d", "PAYX_Paychex_ret_20d", "XLB_Materials_zscore_60d", "US5Y_Rate_ret_5d", "BTI_BritishAmerican_ret_5d", "NWL_Newell_ret_20d", "AMT_AmericanTower_ret_1d", "XOM_ret_1d", "BTI_BritishAmerican_ret_20d", "CLX_Clorox_vol_20d", "HangSeng_HK_ret_1d", "Brent_Oil_FRED_ret_20d", "EWJ_Japan_vol_20d", "CTAS_Cintas_vol_20d", "VOD_Vodafone_zscore_60d", "EWY_Korea_ret_20d", "LOW_Lowes_ret_20d", "EMR_Emerson_ret_20d", "EWG_Germany_vol_20d", "NEE_NextEra_ret_20d", "CPB_CampbellSoup_zscore_60d"], "is_new": true}, {"model_id": "new_h1_GLOBAL_GradientBoosting_N25_t1", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 1, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "3M_ret_5d", "JNJ_ret_1d", "ES_Evergy_ret_1d", "EWY_Korea_ret_20d", "AXP_Amex_ret_20d", "EXC_Exelon_ret_1d", "ITT_ITTInc_ret_5d", "US1Y_Rate_ret_5d", "SLB_Schlumberger_ret_5d", "CPB_CampbellSoup_ret_20d", "M_Macys_vol_20d", "IYR_US_REIT2_zscore_60d", "AMT_AmericanTower_ret_1d", "ENB_EnbridgeInc_ret_1d", "LOW_Lowes_ret_5d", "HD_ret_1d", "HangSeng_HK_ret_5d", "SCHW_Schwab_ret_5d", "DIS_vol_20d", "FedFunds_zscore_60d", "DAX_Germany_vol_20d", "heston_ev_h3", "Nikkei_Japan_vol_20d"], "is_new": true}, {"model_id": "new_h1_GLOBAL_GradientBoosting_N25_t2", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 1, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "GILD_Gilead_ret_20d", "AORD_AUS_zscore_60d", "US3M_Rate_zscore_60d", "NOC_Northrop_ret_20d", "M_Macys_vol_20d", "BTI_BritishAmerican_ret_20d", "PPL_PPL_ret_1d", "SBUX_ret_5d", "hmm_p_stress", "SPY_zscore_60d", "DAX_Germany_zscore_60d", "EXC_Exelon_zscore_60d", "VVIX_ret_20d", "TM_Telephone_ret_1d", "US3Y_Rate_ret_5d", "TXN_vol_20d", "AVB_AvalonBay_zscore_60d", "XLK_Tech_zscore_60d", "TM_Telephone_vol_20d", "3M_ret_5d", "spx_abs_ret_max_5d", "PG_ret_20d", "US1Y_Rate_ret_20d"], "is_new": true}, {"model_id": "new_h1_GLOBAL_GradientBoosting_N25_t3", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 1, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "PAYX_Paychex_ret_20d", "GE_ret_1d", "NFCI_ret_5d", "SPY_zscore_60d", "CPB_CampbellSoup_vol_20d", "EWM_Malaysia_ret_1d", "IWM_SmallCap_vol_20d", "XOM_ret_1d", "ES_Evergy_ret_1d", "MS_MorganStanley_ret_5d", "XOM_ret_20d", "MSTR_Bitcoin3_ret_1d", "FedFunds_zscore_60d", "TXN_vol_20d", "NWL_Newell_ret_20d", "TM_Telephone_ret_1d", "EWY_Korea_ret_20d", "US6M_Rate_ret_20d", "MS_MorganStanley_zscore_60d", "T_ret_1d", "ORCL_zscore_60d", "IBEX_Spain_ret_20d", "SLB_Schlumberger_ret_1d"], "is_new": true}, {"model_id": "new_h1_GLOBAL_GradientBoosting_N25_t4", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 1, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "JNJ_ret_1d", "VOD_Vodafone_zscore_60d", "EXC_Exelon_zscore_60d", "INTC_ret_1d", "AXP_Amex_vol_20d", "CTAS_Cintas_vol_20d", "DAX_Germany_vol_20d", "XLV_Health_zscore_60d", "DE_Deere_vol_20d", "WTI_Oil_FRED_zscore_60d", "EQR_Equity_ret_1d", "BDX_Becton_Dickinson_ret_20d", "Retail_Sales_zscore_60d", "LOW_Lowes_ret_5d", "EWY_Korea_ret_20d", "MSTR_Bitcoin3_ret_20d", "AXP_Amex_ret_20d", "LMT_LockheedMartin_ret_1d", "Core_CPI_zscore_60d", "Brent_Oil_FRED_ret_20d", "QQQ_vol_20d", "CCI_CrownCastle_vol_20d", "NWL_Newell_ret_20d"], "is_new": true}, {"model_id": "new_h1_GLOBAL_GradientBoosting_N25_t5", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 1, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWH_HongKong_ret_5d", "NVDA_vol_20d", "INTC_ret_1d", "EWY_Korea_ret_20d", "NFCI_ret_5d", "heston_var_ev_h7", "DAX_Germany_zscore_60d", "CPB_CampbellSoup_ret_5d", "SO_SouthernCo_ret_5d", "US6M_Rate_ret_20d", "EXC_Exelon_zscore_60d", "IWM_SmallCap_vol_20d", "MRK_Merck_zscore_60d", "WTI_Oil_FRED_zscore_60d", "BDX_Becton_Dickinson_ret_20d", "MSTR_Bitcoin3_ret_20d", "SBUX_vol_20d", "3M_vol_20d", "AXP_Amex_vol_20d", "AMZN_ret_5d", "US3Y_Rate_ret_5d", "HangSeng_HK_ret_5d", "DE_Deere_vol_20d"], "is_new": true}, {"model_id": "new_h1_GLOBAL_GradientBoosting_N25_t6", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 1, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "HUM_Humana_ret_5d", "MSTR_Bitcoin3_ret_1d", "vix_acceleration_1d", "GILD_Gilead_ret_20d", "US1Y_Rate_ret_5d", "XLV_Health_zscore_60d", "ORCL_vol_20d", "Nikkei_Japan_zscore_60d", "SJM_JM_Smucker_ret_1d", "NFCI_ret_5d", "AXP_Amex_vol_20d", "IWM_SmallCap_vol_20d", "heston_var_ev_h7", "FedFunds_zscore_60d", "EMR_Emerson_ret_20d", "AMD_ret_5d", "IBEX_Spain_ret_20d", "INTC_ret_1d", "NWL_Newell_ret_20d", "LUV_SouthwestAir_ret_5d", "SBUX_zscore_60d", "PAYX_Paychex_ret_20d", "JNJ_ret_1d"], "is_new": true}, {"model_id": "new_h1_GLOBAL_GradientBoosting_N25_t7", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 1, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "INTC_ret_5d", "EWL_Switzerland_zscore_60d", "EWA_Australia_zscore_60d", "SLB_Schlumberger_ret_1d", "XLF_Fin_vol_20d", "AORD_AUS_zscore_60d", "NWL_Newell_ret_20d", "DE_Deere_ret_5d", "EWS_Singapore_ret_5d", "MO_AltriaMG_ret_1d", "EXC_Exelon_zscore_60d", "Michigan_Sentiment_ret_20d", "MSTR_Bitcoin3_ret_20d", "heston_var_ev_h5", "ES_Evergy_ret_1d", "NVDA_vol_20d", "Industrial_Production_zscore_60d", "HD_zscore_60d", "BA_ret_1d", "CPB_CampbellSoup_zscore_60d", "MS_MorganStanley_zscore_60d", "IYR_US_REIT2_zscore_60d", "LLY_zscore_60d"], "is_new": true}, {"model_id": "new_h1_GLOBAL_GradientBoosting_N30_t0", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 1, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EMR_Emerson_ret_20d", "ES_Evergy_ret_1d", "Core_PCE_zscore_60d", "EOG_EOGResources_vol_20d", "DOW_Price_zscore_60d", "HUM_Humana_ret_5d", "EQR_Equity_ret_1d", "PAYX_Paychex_zscore_60d", "Nikkei_Japan_vol_20d", "Retail_Sales_zscore_60d", "QQQ_vol_20d", "IBEX_Spain_ret_20d", "M_Macys_vol_20d", "DE_Deere_vol_20d", "INTC_ret_1d", "XLK_Tech_zscore_60d", "BTI_BritishAmerican_ret_5d", "MS_MorganStanley_ret_1d", "AORD_AUS_zscore_60d", "CPB_CampbellSoup_vol_20d", "CCI_CrownCastle_vol_20d", "EQIX_Equinix_ret_5d", "HD_ret_1d", "CPB_CampbellSoup_ret_5d", "CMCSA_ret_1d", "MRK_Merck_zscore_60d", "SJM_JM_Smucker_ret_5d", "EWY_Korea_zscore_60d"], "is_new": true}, {"model_id": "new_h1_GLOBAL_GradientBoosting_N30_t1", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 1, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "TM_Telephone_ret_1d", "AMD_ret_5d", "CPB_CampbellSoup_zscore_60d", "QQQ_vol_20d", "VOD_Vodafone_zscore_60d", "SPY_zscore_60d", "MS_MorganStanley_zscore_60d", "SBUX_ret_5d", "SCHW_Schwab_ret_5d", "US30Y_Rate_ret_20d", "DAX_Germany_zscore_60d", "EWC_Canada_zscore_60d", "CCI_CrownCastle_vol_20d", "EOG_EOGResources_vol_20d", "US5Y_Rate_ret_5d", "VVIX_ret_20d", "LOW_Lowes_ret_20d", "PAYX_Paychex_ret_20d", "vix_acceleration_1d", "GE_ret_1d", "PAYX_Paychex_vol_20d", "DE_Deere_vol_20d", "NEE_NextEra_ret_20d", "FedFunds_zscore_60d", "ENB_EnbridgeInc_ret_1d", "NWL_Newell_ret_20d", "Brent_Oil_FRED_ret_5d", "DIS_vol_20d"], "is_new": true}, {"model_id": "new_h1_GLOBAL_GradientBoosting_N30_t2", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 1, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWG_Germany_vol_20d", "TED_Spread_zscore_60d", "EWY_Korea_zscore_60d", "EWM_Malaysia_vol_20d", "Retail_Sales_zscore_60d", "JNJ_ret_1d", "spx_momentum_3d", "Nikkei_Japan_zscore_60d", "LLY_zscore_60d", "VOD_Vodafone_zscore_60d", "MSTR_Bitcoin3_ret_5d", "TGT_Target_zscore_60d", "AXP_Amex_vol_20d", "PG_ret_20d", "US1Y_Rate_ret_20d", "LMT_LockheedMartin_ret_1d", "EOG_EOGResources_ret_5d", "MO_AltriaMG_ret_1d", "FedFunds_zscore_60d", "SO_SouthernCo_ret_5d", "EOG_EOGResources_vol_20d", "IBEX_Spain_ret_20d", "DAX_Germany_zscore_60d", "DAX_Germany_vol_20d", "US1Y_Rate_ret_5d", "EWM_Malaysia_zscore_60d", "SJM_JM_Smucker_ret_5d", "IWM_SmallCap_vol_20d"], "is_new": true}, {"model_id": "new_h1_GLOBAL_GradientBoosting_N30_t3", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 1, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "ASX_Australia_ret_5d", "Nikkei_Japan_vol_20d", "EXC_Exelon_ret_1d", "HangSeng_HK_vol_20d", "spx_abs_ret_max_5d", "MO_AltriaMG_ret_1d", "HD_ret_5d", "PLD_Prologis_ret_5d", "Industrial_Production_zscore_60d", "DIS_vol_20d", "EWC_Canada_zscore_60d", "Brent_Oil_FRED_ret_5d", "CTAS_Cintas_vol_20d", "DHR_ret_1d", "EWA_Australia_zscore_60d", "EFFR_ret_1d", "PAYX_Paychex_vol_20d", "heston_var_ev_h7", "TM_Telephone_ret_1d", "SBUX_zscore_60d", "CI_Cigna_vol_20d", "PAYX_Paychex_zscore_60d", "ORCL_zscore_60d", "MS_MorganStanley_zscore_60d", "EWG_Germany_vol_20d", "EWS_Singapore_ret_5d", "HUM_Humana_ret_5d", "ENB_EnbridgeInc_ret_1d"], "is_new": true}, {"model_id": "new_h1_GLOBAL_GradientBoosting_N30_t4", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 1, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "US1Y_Rate_ret_5d", "TM_Telephone_ret_1d", "MSTR_Bitcoin3_ret_20d", "EOG_EOGResources_ret_5d", "Brent_Oil_FRED_ret_20d", "QQQ_vol_20d", "LMT_LockheedMartin_vol_20d", "AMD_ret_5d", "TM_Telephone_vol_20d", "BLK_BlackRock_zscore_60d", "BTI_BritishAmerican_ret_5d", "XLF_Fin_vol_20d", "MS_MorganStanley_ret_1d", "EWY_Korea_ret_20d", "AVB_AvalonBay_zscore_60d", "EXC_Exelon_ret_1d", "IYM_BasicMaterials_ret_20d", "EWA_Australia_zscore_60d", "ASX_Australia_ret_5d", "MSTR_Bitcoin3_ret_5d", "EWM_Malaysia_vol_20d", "IWM_SmallCap_vol_20d", "TXN_vol_20d", "SJM_JM_Smucker_ret_5d", "DIS_vol_20d", "GD_GeneralDynamics_zscore_60d", "VRP_ma5", "XOM_ret_20d"], "is_new": true}, {"model_id": "new_h1_GLOBAL_GradientBoosting_N30_t5", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 1, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "heston_ev_h3", "CPB_CampbellSoup_vol_20d", "SO_SouthernCo_ret_5d", "BTI_BritishAmerican_ret_5d", "MS_MorganStanley_zscore_60d", "ORCL_zscore_60d", "QQQ_vol_20d", "SLB_Schlumberger_ret_1d", "HD_ret_5d", "spx_vol_5d", "spx_momentum_3d", "NWL_Newell_ret_20d", "LLY_zscore_60d", "EOG_EOGResources_vol_20d", "HD_ret_1d", "EXC_Exelon_ret_1d", "XLK_Tech_zscore_60d", "LOW_Lowes_ret_5d", "EXC_Exelon_zscore_60d", "FedFunds_zscore_60d", "Nikkei_Japan_vol_20d", "BA_ret_1d", "TGT_Target_zscore_60d", "EWH_HongKong_ret_5d", "US7Y_Rate_ret_20d", "XLB_Materials_zscore_60d", "BTI_BritishAmerican_ret_20d", "heston_var_ev_h7"], "is_new": true}, {"model_id": "new_h1_GLOBAL_GradientBoosting_N30_t6", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 1, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "AMD_ret_5d", "heston_var_ev_h5", "hmm_p_stress", "ITT_ITTInc_ret_5d", "EWQ_France_zscore_60d", "vix_acceleration_1d", "VVIX_ret_20d", "DAX_Germany_vol_20d", "DIS_vol_20d", "TM_Telephone_ret_1d", "PCAR_PaccarInc_ret_5d", "SJM_JM_Smucker_ret_5d", "SO_SouthernCo_ret_5d", "EXC_Exelon_ret_1d", "heston_ev_h3", "MSTR_Bitcoin3_ret_5d", "US3M_Rate_zscore_60d", "US6M_Rate_ret_20d", "NWL_Newell_ret_20d", "VOD_Vodafone_zscore_60d", "TED_Spread_zscore_60d", "XOM_ret_1d", "EFFR_ret_1d", "BLK_BlackRock_zscore_60d", "HangSeng_HK_vol_20d", "INTC_ret_1d", "SBUX_ret_5d", "DE_Deere_vol_20d"], "is_new": true}, {"model_id": "new_h1_GLOBAL_GradientBoosting_N30_t7", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 1, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "DHR_ret_1d", "EWY_Korea_zscore_60d", "MSTR_Bitcoin3_ret_1d", "EWM_Malaysia_zscore_60d", "ASX_Australia_vol_20d", "CLX_Clorox_vol_20d", "WTI_Oil_FRED_zscore_60d", "TM_Telephone_vol_20d", "MSTR_Bitcoin3_ret_20d", "vix_acceleration_1d", "EOG_EOGResources_vol_20d", "GD_GeneralDynamics_zscore_60d", "DE_Deere_ret_5d", "EWG_Germany_vol_20d", "US1Y_Rate_ret_5d", "EWM_Malaysia_ret_1d", "VRP_ma5", "PAYX_Paychex_zscore_60d", "EMR_Emerson_ret_20d", "T10Y2Y_Spread_ret_5d", "EWQ_France_ret_20d", "XLF_Fin_vol_20d", "US3M_Rate_zscore_60d", "CPB_CampbellSoup_vol_20d", "DOW_Price_zscore_60d", "CTAS_Cintas_vol_20d", "T_ret_1d", "SJM_JM_Smucker_ret_1d"], "is_new": true}, {"model_id": "new_h1_GLOBAL_RandomForest_N5_t0", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 1, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EOG_EOGResources_vol_20d", "SPY_zscore_60d", "SO_SouthernCo_ret_5d"], "is_new": true}, {"model_id": "new_h1_GLOBAL_RandomForest_N5_t1", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 1, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "DAX_Germany_vol_20d", "AXP_Amex_ret_20d", "Core_CPI_zscore_60d"], "is_new": true}, {"model_id": "new_h1_GLOBAL_RandomForest_N5_t2", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 1, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "3M_ret_5d", "AMGN_Amgen_ret_1d", "BTI_BritishAmerican_ret_20d"], "is_new": true}, {"model_id": "new_h1_GLOBAL_RandomForest_N5_t3", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 1, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "MSTR_Bitcoin3_ret_20d", "Core_CPI_zscore_60d", "AXP_Amex_ret_20d"], "is_new": true}, {"model_id": "new_h1_GLOBAL_RandomForest_N5_t4", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 1, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "AMD_ret_1d", "T_ret_1d", "CI_Cigna_vol_20d"], "is_new": true}, {"model_id": "new_h1_GLOBAL_RandomForest_N5_t5", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 1, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "CMCSA_ret_1d", "HD_ret_5d", "BLK_BlackRock_zscore_60d"], "is_new": true}, {"model_id": "new_h1_GLOBAL_RandomForest_N5_t6", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 1, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "hmm_p_stress", "EWS_Singapore_ret_5d", "heston_var_ev_h5"], "is_new": true}, {"model_id": "new_h1_GLOBAL_RandomForest_N5_t7", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 1, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "NVDA_vol_20d", "T_ret_1d", "heston_var_ev_h5"], "is_new": true}, {"model_id": "new_h1_GLOBAL_RandomForest_N8_t0", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 1, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "NEE_NextEra_ret_20d", "VOD_Vodafone_zscore_60d", "INTC_ret_1d", "CPB_CampbellSoup_ret_5d", "ASX_Australia_ret_5d", "3M_ret_5d"], "is_new": true}, {"model_id": "new_h1_GLOBAL_RandomForest_N8_t1", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 1, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EQR_Equity_ret_1d", "3M_ret_5d", "MO_AltriaMG_ret_1d", "HD_ret_1d", "IYM_BasicMaterials_ret_20d", "VOD_Vodafone_zscore_60d"], "is_new": true}, {"model_id": "new_h1_GLOBAL_RandomForest_N8_t2", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 1, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "AMT_AmericanTower_ret_1d", "AXP_Amex_ret_20d", "ORCL_vol_20d", "Nikkei_Japan_zscore_60d", "vix_mean_abs_ret_5d", "MS_MorganStanley_ret_1d"], "is_new": true}, {"model_id": "new_h1_GLOBAL_RandomForest_N8_t3", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 1, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "AMD_ret_5d", "heston_var_ev_h7", "MS_MorganStanley_ret_5d", "MO_AltriaMG_ret_1d", "HangSeng_HK_ret_1d", "CLX_Clorox_vol_20d"], "is_new": true}, {"model_id": "new_h1_GLOBAL_RandomForest_N8_t4", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 1, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "3M_ret_5d", "PAYX_Paychex_vol_20d", "TM_Telephone_vol_20d", "ITT_ITTInc_ret_5d", "CPB_CampbellSoup_ret_5d", "M_Macys_vol_20d"], "is_new": true}, {"model_id": "new_h1_GLOBAL_RandomForest_N8_t5", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 1, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "T10Y2Y_Spread_ret_5d", "XLV_Health_zscore_60d", "AMD_ret_5d", "MO_AltriaMG_ret_1d", "US30Y_Rate_ret_20d", "EWA_Australia_zscore_60d"], "is_new": true}, {"model_id": "new_h1_GLOBAL_RandomForest_N8_t6", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 1, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "US1Y_Rate_ret_20d", "DHR_vol_20d", "MSTR_Bitcoin3_ret_1d", "TGT_Target_zscore_60d", "US3M_Rate_zscore_60d", "US3M_Rate_vol_20d"], "is_new": true}, {"model_id": "new_h1_GLOBAL_RandomForest_N8_t7", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 1, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "XLF_Fin_vol_20d", "PFE_ret_1d", "AMD_ret_5d", "TED_Spread_zscore_60d", "EWM_Malaysia_zscore_60d", "TM_Telephone_vol_20d"], "is_new": true}, {"model_id": "new_h1_GLOBAL_RandomForest_N10_t0", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 1, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "XLV_Health_zscore_60d", "Brent_Oil_FRED_ret_20d", "LUV_SouthwestAir_ret_5d", "AORD_AUS_zscore_60d", "MO_AltriaMG_ret_1d", "VRP_ma5", "Nikkei_Japan_vol_20d", "EQR_Equity_ret_1d"], "is_new": true}, {"model_id": "new_h1_GLOBAL_RandomForest_N10_t1", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 1, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "MSTR_Bitcoin3_ret_1d", "AMT_AmericanTower_ret_1d", "VVIX_ret_20d", "EWQ_France_ret_20d", "DAX_Germany_zscore_60d", "CLX_Clorox_vol_20d", "DE_Deere_ret_5d", "Nikkei_Japan_zscore_60d"], "is_new": true}, {"model_id": "new_h1_GLOBAL_RandomForest_N10_t2", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 1, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "MSTR_Bitcoin3_ret_20d", "AMD_ret_5d", "HangSeng_HK_ret_5d", "CCI_CrownCastle_vol_20d", "US1Y_Rate_ret_20d", "3M_vol_20d", "US3Y_Rate_ret_5d", "MO_AltriaMG_ret_1d"], "is_new": true}, {"model_id": "new_h1_GLOBAL_RandomForest_N10_t3", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 1, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "IYR_US_REIT2_zscore_60d", "SLB_Schlumberger_ret_5d", "INTC_ret_1d", "Core_PCE_zscore_60d", "PFE_ret_1d", "SBUX_vol_20d", "PCAR_PaccarInc_ret_5d", "HD_zscore_60d"], "is_new": true}, {"model_id": "new_h1_GLOBAL_RandomForest_N10_t4", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 1, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "HUM_Humana_ret_5d", "MSTR_Bitcoin3_ret_5d", "XLB_Materials_zscore_60d", "ES_Evergy_ret_1d", "Brent_Oil_FRED_ret_5d", "CPB_CampbellSoup_ret_20d", "TM_Telephone_vol_20d", "heston_var_ev_h7"], "is_new": true}, {"model_id": "new_h1_GLOBAL_RandomForest_N10_t5", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 1, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "XLF_Fin_vol_20d", "VRP_ma5", "Michigan_Sentiment_ret_20d", "DHR_ret_1d", "ORCL_vol_20d", "GD_GeneralDynamics_zscore_60d", "EQIX_Equinix_ret_5d", "EMR_Emerson_ret_20d"], "is_new": true}, {"model_id": "new_h1_GLOBAL_RandomForest_N10_t6", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 1, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "PAYX_Paychex_zscore_60d", "BA_ret_1d", "FedFunds_zscore_60d", "LUV_SouthwestAir_ret_5d", "GD_GeneralDynamics_zscore_60d", "DE_Deere_ret_5d", "CPB_CampbellSoup_vol_20d", "US1Y_Rate_ret_20d"], "is_new": true}, {"model_id": "new_h1_GLOBAL_RandomForest_N10_t7", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 1, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "AXP_Amex_ret_20d", "INTC_ret_5d", "ORCL_vol_20d", "SLB_Schlumberger_ret_1d", "DAX_Germany_zscore_60d", "3M_ret_5d", "US5Y_Rate_ret_5d", "DHR_vol_20d"], "is_new": true}, {"model_id": "new_h1_GLOBAL_RandomForest_N12_t0", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 1, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "US5Y_Rate_ret_5d", "AMGN_Amgen_ret_1d", "AXP_Amex_vol_20d", "BTI_BritishAmerican_ret_20d", "gjr_condvar_h1", "M_Macys_vol_20d", "EWL_Switzerland_zscore_60d", "JNJ_ret_1d", "NEE_NextEra_ret_20d", "GILD_Gilead_ret_20d"], "is_new": true}, {"model_id": "new_h1_GLOBAL_RandomForest_N12_t1", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 1, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "DE_Deere_vol_20d", "HD_ret_1d", "PPL_PPL_ret_1d", "EQIX_Equinix_ret_5d", "SJM_JM_Smucker_ret_5d", "EMR_Emerson_ret_20d", "US30Y_Rate_ret_20d", "XLK_Tech_zscore_60d", "IWM_SmallCap_vol_20d", "3M_vol_20d"], "is_new": true}, {"model_id": "new_h1_GLOBAL_RandomForest_N12_t2", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 1, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EMR_Emerson_ret_20d", "FedFunds_zscore_60d", "PAYX_Paychex_zscore_60d", "EWQ_France_zscore_60d", "CMCSA_ret_1d", "DE_Deere_vol_20d", "NVDA_vol_20d", "Brent_Oil_FRED_ret_20d", "MS_MorganStanley_ret_1d", "TM_Telephone_vol_20d"], "is_new": true}, {"model_id": "new_h1_GLOBAL_RandomForest_N12_t3", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 1, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "BLK_BlackRock_zscore_60d", "LMT_LockheedMartin_ret_1d", "US30Y_Rate_ret_20d", "heston_var_ev_h5", "CMCSA_ret_1d", "Core_CPI_zscore_60d", "US5Y_Rate_ret_5d", "MRK_Merck_zscore_60d", "3M_ret_5d", "XLK_Tech_zscore_60d"], "is_new": true}, {"model_id": "new_h1_GLOBAL_RandomForest_N12_t4", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 1, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "INTC_ret_1d", "ES_Evergy_ret_1d", "Nikkei_Japan_vol_20d", "CCI_CrownCastle_vol_20d", "AXP_Amex_ret_20d", "LOW_Lowes_ret_5d", "EWM_Malaysia_vol_20d", "ORCL_zscore_60d", "BA_ret_1d", "US3M_Rate_zscore_60d"], "is_new": true}, {"model_id": "new_h1_GLOBAL_RandomForest_N12_t5", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 1, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "PG_ret_20d", "Brent_Oil_FRED_ret_20d", "Nikkei_Japan_zscore_60d", "VRP_ma5", "INTC_ret_1d", "AMD_ret_5d", "3M_vol_20d", "SBUX_ret_5d", "EXC_Exelon_ret_1d", "spx_abs_ret_max_5d"], "is_new": true}, {"model_id": "new_h1_GLOBAL_RandomForest_N12_t6", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 1, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWM_Malaysia_vol_20d", "heston_var_ev_h7", "DHR_vol_20d", "US3M_Rate_zscore_60d", "T10Y2Y_Spread_ret_5d", "PLD_Prologis_ret_5d", "NFCI_ret_5d", "VRP_ma5", "US1Y_Rate_ret_5d", "US3Y_Rate_ret_5d"], "is_new": true}, {"model_id": "new_h1_GLOBAL_RandomForest_N12_t7", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 1, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "VVIX_ret_20d", "EXC_Exelon_ret_1d", "HD_ret_1d", "PPL_PPL_ret_1d", "Core_CPI_zscore_60d", "NFCI_ret_5d", "CLX_Clorox_vol_20d", "XOM_ret_1d", "SO_SouthernCo_ret_5d", "DHR_vol_20d"], "is_new": true}, {"model_id": "new_h1_GLOBAL_RandomForest_N15_t0", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 1, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWJ_Japan_vol_20d", "Brent_Oil_FRED_ret_20d", "ASX_Australia_vol_20d", "EXC_Exelon_zscore_60d", "IYR_US_REIT2_zscore_60d", "AXP_Amex_ret_20d", "EFFR_ret_1d", "LMT_LockheedMartin_vol_20d", "US30Y_Rate_ret_20d", "spx_vol_5d", "XLK_Tech_zscore_60d", "Michigan_Sentiment_ret_20d", "EWL_Switzerland_zscore_60d"], "is_new": true}, {"model_id": "new_h1_GLOBAL_RandomForest_N15_t1", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 1, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "HUM_Humana_ret_5d", "EWJ_Japan_vol_20d", "LOW_Lowes_ret_5d", "ORCL_zscore_60d", "EXC_Exelon_ret_1d", "TED_Spread_vol_20d", "EWQ_France_zscore_60d", "Industrial_Production_zscore_60d", "DOW_Price_zscore_60d", "NVDA_vol_20d", "TED_Spread_zscore_60d", "BLK_BlackRock_zscore_60d", "Michigan_Sentiment_ret_20d"], "is_new": true}, {"model_id": "new_h1_GLOBAL_RandomForest_N15_t2", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 1, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "PAYX_Paychex_vol_20d", "EWA_Australia_zscore_60d", "VOD_Vodafone_zscore_60d", "T_ret_1d", "EWL_Switzerland_zscore_60d", "EWY_Korea_zscore_60d", "HangSeng_HK_ret_1d", "TM_Telephone_vol_20d", "SJM_JM_Smucker_ret_5d", "HangSeng_HK_ret_5d", "EQIX_Equinix_ret_5d", "INTC_ret_5d", "EWM_Malaysia_vol_20d"], "is_new": true}, {"model_id": "new_h1_GLOBAL_RandomForest_N15_t3", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 1, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "TM_Telephone_vol_20d", "AXP_Amex_vol_20d", "IYM_BasicMaterials_ret_20d", "IYR_US_REIT2_zscore_60d", "BA_ret_1d", "PAYX_Paychex_vol_20d", "PAYX_Paychex_ret_20d", "GILD_Gilead_ret_20d", "MS_MorganStanley_ret_5d", "LOW_Lowes_ret_5d", "XLY_Disc_vol_20d", "LMT_LockheedMartin_vol_20d", "CCI_CrownCastle_vol_20d"], "is_new": true}, {"model_id": "new_h1_GLOBAL_RandomForest_N15_t4", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 1, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "US3M_Rate_vol_20d", "EQR_Equity_ret_1d", "AMD_ret_1d", "VVIX_ret_20d", "ASX_Australia_ret_5d", "TM_Telephone_vol_20d", "IYR_US_REIT2_zscore_60d", "IWM_SmallCap_vol_20d", "CPB_CampbellSoup_zscore_60d", "AVB_AvalonBay_zscore_60d", "SCHW_Schwab_ret_5d", "gjr_condvar_h1", "IBEX_Spain_ret_20d"], "is_new": true}, {"model_id": "new_h1_GLOBAL_RandomForest_N15_t5", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 1, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "CPB_CampbellSoup_ret_5d", "CI_Cigna_vol_20d", "PG_ret_20d", "SLB_Schlumberger_ret_1d", "MS_MorganStanley_ret_1d", "CPB_CampbellSoup_ret_20d", "WTI_Oil_FRED_zscore_60d", "AMGN_Amgen_ret_1d", "DHR_vol_20d", "IYM_BasicMaterials_ret_20d", "HUM_Humana_ret_5d", "PAYX_Paychex_ret_20d", "BA_ret_1d"], "is_new": true}, {"model_id": "new_h1_GLOBAL_RandomForest_N15_t6", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 1, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "Michigan_Sentiment_ret_20d", "XLY_Disc_vol_20d", "MO_AltriaMG_ret_1d", "EWM_Malaysia_zscore_60d", "INTC_ret_1d", "GD_GeneralDynamics_zscore_60d", "CI_Cigna_vol_20d", "EQIX_Equinix_ret_5d", "3M_ret_5d", "vix_acceleration_1d", "DE_Deere_vol_20d", "EQR_Equity_ret_1d", "ORCL_zscore_60d"], "is_new": true}, {"model_id": "new_h1_GLOBAL_RandomForest_N15_t7", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 1, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EFFR_vol_20d", "NOC_Northrop_ret_20d", "AXP_Amex_vol_20d", "CCI_CrownCastle_vol_20d", "INTC_ret_5d", "CTAS_Cintas_vol_20d", "Industrial_Production_zscore_60d", "EWJ_Japan_vol_20d", "EQR_Equity_ret_1d", "LLY_zscore_60d", "spx_abs_ret_max_5d", "VOD_Vodafone_zscore_60d", "EXC_Exelon_zscore_60d"], "is_new": true}, {"model_id": "new_h1_GLOBAL_RandomForest_N20_t0", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 1, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "HD_ret_20d", "AXP_Amex_ret_20d", "SCHW_Schwab_ret_5d", "CPB_CampbellSoup_ret_5d", "PAYX_Paychex_ret_20d", "EWA_Australia_ret_1d", "DE_Deere_ret_5d", "PG_ret_20d", "VVIX_ret_20d", "T_ret_1d", "AXP_Amex_vol_20d", "LMT_LockheedMartin_vol_20d", "VRP_ma5", "LUV_SouthwestAir_ret_5d", "heston_var_ev_h5", "AORD_AUS_zscore_60d", "EWM_Malaysia_zscore_60d", "vix_acceleration_1d"], "is_new": true}, {"model_id": "new_h1_GLOBAL_RandomForest_N20_t1", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 1, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "HD_ret_5d", "XOM_ret_1d", "EWL_Switzerland_vol_20d", "HangSeng_HK_ret_5d", "AMD_ret_5d", "EWG_Germany_vol_20d", "US6M_Rate_ret_20d", "Core_PCE_zscore_60d", "3M_ret_5d", "HD_ret_1d", "LMT_LockheedMartin_ret_1d", "DAX_Germany_zscore_60d", "ASX_Australia_ret_5d", "EWJ_Japan_vol_20d", "Nikkei_Japan_vol_20d", "AXP_Amex_ret_20d", "SLB_Schlumberger_ret_1d", "XLK_Tech_zscore_60d"], "is_new": true}, {"model_id": "new_h1_GLOBAL_RandomForest_N20_t2", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 1, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "TM_Telephone_vol_20d", "HangSeng_HK_ret_1d", "XOM_ret_20d", "XLV_Health_zscore_60d", "CPB_CampbellSoup_ret_20d", "EWL_Switzerland_zscore_60d", "spx_abs_ret_max_5d", "Industrial_Production_zscore_60d", "EOG_EOGResources_vol_20d", "TXN_vol_20d", "LOW_Lowes_ret_20d", "LLY_zscore_60d", "EWS_Singapore_ret_5d", "MRK_Merck_zscore_60d", "US3M_Rate_zscore_60d", "JNJ_ret_1d", "DIS_vol_20d", "AXP_Amex_ret_20d"], "is_new": true}, {"model_id": "new_h1_GLOBAL_RandomForest_N20_t3", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 1, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "BLK_BlackRock_zscore_60d", "NEE_NextEra_ret_20d", "SBUX_vol_20d", "US3M_Rate_zscore_60d", "US5Y_Rate_ret_5d", "AMD_ret_5d", "heston_var_ev_h5", "AMGN_Amgen_ret_1d", "MS_MorganStanley_zscore_60d", "US3Y_Rate_ret_5d", "Nikkei_Japan_vol_20d", "T_ret_1d", "Brent_Oil_FRED_ret_20d", "TXN_vol_20d", "MS_MorganStanley_ret_1d", "ORCL_zscore_60d", "Retail_Sales_zscore_60d", "FedFunds_zscore_60d"], "is_new": true}, {"model_id": "new_h1_GLOBAL_RandomForest_N20_t4", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 1, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "DAX_Germany_zscore_60d", "EWL_Switzerland_vol_20d", "spx_vol_5d", "EWA_Australia_zscore_60d", "US3M_Rate_zscore_60d", "HD_ret_5d", "HangSeng_HK_vol_20d", "US6M_Rate_ret_20d", "MRK_Merck_zscore_60d", "EWM_Malaysia_ret_1d", "AMT_AmericanTower_ret_1d", "3M_ret_5d", "MS_MorganStanley_ret_5d", "SBUX_zscore_60d", "PFE_ret_1d", "MSTR_Bitcoin3_ret_1d", "EFFR_vol_20d", "IYR_US_REIT2_zscore_60d"], "is_new": true}, {"model_id": "new_h1_GLOBAL_RandomForest_N20_t5", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 1, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "heston_ev_h3", "LMT_LockheedMartin_ret_1d", "AMD_ret_1d", "Brent_Oil_FRED_ret_20d", "SCHW_Schwab_ret_5d", "EWJ_Japan_vol_20d", "MSTR_Bitcoin3_ret_20d", "INTC_ret_1d", "EXC_Exelon_ret_1d", "DE_Deere_ret_5d", "MS_MorganStanley_ret_5d", "ES_Evergy_ret_1d", "HD_ret_5d", "CPB_CampbellSoup_zscore_60d", "PFE_ret_1d", "TM_Telephone_ret_1d", "US7Y_Rate_ret_20d", "BLK_BlackRock_zscore_60d"], "is_new": true}, {"model_id": "new_h1_GLOBAL_RandomForest_N20_t6", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 1, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWL_Switzerland_zscore_60d", "HangSeng_HK_vol_20d", "IYM_BasicMaterials_ret_20d", "CPB_CampbellSoup_ret_20d", "BTI_BritishAmerican_ret_5d", "AMD_ret_5d", "ASX_Australia_ret_5d", "US3Y_Rate_ret_5d", "AVB_AvalonBay_zscore_60d", "heston_var_ev_h5", "NVDA_vol_20d", "AXP_Amex_ret_20d", "LOW_Lowes_ret_20d", "EWS_Singapore_ret_5d", "3M_vol_20d", "heston_var_ev_h3", "TM_Telephone_vol_20d", "US1Y_Rate_ret_5d"], "is_new": true}, {"model_id": "new_h1_GLOBAL_RandomForest_N20_t7", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 1, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "DOW_Price_zscore_60d", "Core_CPI_zscore_60d", "CLX_Clorox_vol_20d", "MSTR_Bitcoin3_ret_5d", "MS_MorganStanley_ret_5d", "PAYX_Paychex_ret_20d", "EWL_Switzerland_zscore_60d", "DE_Deere_vol_20d", "XLV_Health_zscore_60d", "3M_vol_20d", "HD_ret_20d", "IYR_US_REIT2_zscore_60d", "PCAR_PaccarInc_ret_5d", "US3M_Rate_vol_20d", "US1Y_Rate_ret_5d", "TXN_vol_20d", "PAYX_Paychex_zscore_60d", "ITT_ITTInc_ret_5d"], "is_new": true}, {"model_id": "new_h1_GLOBAL_RandomForest_N25_t0", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 1, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "VRP_ma5", "vix_mean_abs_ret_5d", "NWL_Newell_ret_20d", "IBEX_Spain_ret_20d", "ENB_EnbridgeInc_ret_1d", "MS_MorganStanley_zscore_60d", "AXP_Amex_ret_20d", "EWM_Malaysia_zscore_60d", "EWQ_France_zscore_60d", "NVDA_vol_20d", "EQIX_Equinix_ret_5d", "AMD_ret_1d", "DHR_vol_20d", "PFE_ret_1d", "MO_AltriaMG_ret_1d", "LMT_LockheedMartin_vol_20d", "T10Y2Y_Spread_ret_5d", "ES_Evergy_ret_1d", "Brent_Oil_FRED_ret_5d", "EXC_Exelon_ret_1d", "PG_ret_20d", "vix_acceleration_1d", "AMZN_ret_5d"], "is_new": true}, {"model_id": "new_h1_GLOBAL_RandomForest_N25_t1", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 1, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "Core_CPI_zscore_60d", "CMCSA_ret_1d", "LLY_zscore_60d", "EMR_Emerson_ret_20d", "EWM_Malaysia_ret_1d", "T_ret_1d", "3M_vol_20d", "EQR_Equity_ret_1d", "EWA_Australia_ret_1d", "Nikkei_Japan_zscore_60d", "EOG_EOGResources_ret_5d", "CPB_CampbellSoup_ret_5d", "XLF_Fin_vol_20d", "gjr_condvar_h1", "BTI_BritishAmerican_ret_5d", "LMT_LockheedMartin_vol_20d", "TED_Spread_vol_20d", "DAX_Germany_vol_20d", "T10Y2Y_Spread_ret_5d", "US3Y_Rate_ret_5d", "SBUX_zscore_60d", "heston_var_ev_h7", "LUV_SouthwestAir_ret_5d"], "is_new": true}, {"model_id": "new_h1_GLOBAL_RandomForest_N25_t2", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 1, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "CTAS_Cintas_vol_20d", "SJM_JM_Smucker_ret_5d", "ORCL_vol_20d", "EWJ_Japan_vol_20d", "TM_Telephone_ret_1d", "MRK_Merck_zscore_60d", "IYR_US_REIT2_zscore_60d", "T_ret_1d", "heston_var_ev_h5", "CCI_CrownCastle_vol_20d", "VOD_Vodafone_zscore_60d", "INTC_ret_1d", "US3M_Rate_zscore_60d", "PG_ret_20d", "PPL_PPL_ret_1d", "ENB_EnbridgeInc_ret_1d", "CPB_CampbellSoup_ret_20d", "MSTR_Bitcoin3_ret_20d", "US1Y_Rate_ret_5d", "SPY_zscore_60d", "AXP_Amex_ret_20d", "AXP_Amex_vol_20d", "Industrial_Production_zscore_60d"], "is_new": true}, {"model_id": "new_h1_GLOBAL_RandomForest_N25_t3", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 1, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "T_ret_1d", "MSTR_Bitcoin3_ret_1d", "XLF_Fin_vol_20d", "EFFR_ret_1d", "SLB_Schlumberger_ret_5d", "US6M_Rate_ret_20d", "Retail_Sales_zscore_60d", "EOG_EOGResources_vol_20d", "spx_abs_ret_max_5d", "GILD_Gilead_ret_20d", "IYM_BasicMaterials_ret_20d", "IBEX_Spain_ret_20d", "BA_ret_1d", "CPB_CampbellSoup_ret_20d", "SO_SouthernCo_ret_5d", "DHR_ret_1d", "EOG_EOGResources_ret_5d", "heston_var_ev_h7", "EMR_Emerson_ret_20d", "3M_ret_5d", "XOM_ret_1d", "EQIX_Equinix_ret_5d", "US1Y_Rate_ret_20d"], "is_new": true}, {"model_id": "new_h1_GLOBAL_RandomForest_N25_t4", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 1, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "DE_Deere_vol_20d", "spx_vol_5d", "EFFR_ret_1d", "BLK_BlackRock_zscore_60d", "EWY_Korea_zscore_60d", "CLX_Clorox_vol_20d", "HD_ret_1d", "heston_var_ev_h7", "SBUX_vol_20d", "heston_var_ev_h5", "EWJ_Japan_vol_20d", "DE_Deere_ret_5d", "TED_Spread_vol_20d", "CTAS_Cintas_vol_20d", "BTI_BritishAmerican_ret_20d", "NVDA_vol_20d", "LOW_Lowes_ret_5d", "US5Y_Rate_ret_5d", "EWC_Canada_zscore_60d", "AMD_ret_1d", "DAX_Germany_zscore_60d", "AMD_ret_5d", "VRP_ma5"], "is_new": true}, {"model_id": "new_h1_GLOBAL_RandomForest_N25_t5", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 1, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "DOW_Price_zscore_60d", "EWL_Switzerland_zscore_60d", "DE_Deere_ret_5d", "spx_vol_5d", "ORCL_zscore_60d", "SJM_JM_Smucker_ret_1d", "LOW_Lowes_ret_20d", "DAX_Germany_zscore_60d", "CPB_CampbellSoup_vol_20d", "EWS_Singapore_ret_5d", "MRK_Merck_zscore_60d", "EWA_Australia_zscore_60d", "AVB_AvalonBay_zscore_60d", "spx_momentum_3d", "SJM_JM_Smucker_ret_5d", "CMCSA_ret_1d", "NOC_Northrop_ret_20d", "IYR_US_REIT2_zscore_60d", "AXP_Amex_vol_20d", "NEE_NextEra_ret_20d", "EWY_Korea_zscore_60d", "MS_MorganStanley_zscore_60d", "EWC_Canada_zscore_60d"], "is_new": true}, {"model_id": "new_h1_GLOBAL_RandomForest_N25_t6", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 1, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "ORCL_zscore_60d", "PCAR_PaccarInc_ret_5d", "PFE_ret_1d", "LOW_Lowes_ret_20d", "MS_MorganStanley_zscore_60d", "US7Y_Rate_ret_20d", "CPB_CampbellSoup_ret_20d", "XLK_Tech_zscore_60d", "TM_Telephone_ret_1d", "EWM_Malaysia_ret_1d", "DE_Deere_ret_5d", "Brent_Oil_FRED_ret_20d", "AMT_AmericanTower_ret_1d", "EMR_Emerson_ret_20d", "WTI_Oil_FRED_zscore_60d", "CLX_Clorox_vol_20d", "NOC_Northrop_ret_20d", "XOM_ret_20d", "PPL_PPL_ret_1d", "HangSeng_HK_ret_1d", "MSTR_Bitcoin3_ret_1d", "AVB_AvalonBay_zscore_60d", "EFFR_vol_20d"], "is_new": true}, {"model_id": "new_h1_GLOBAL_RandomForest_N25_t7", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 1, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "FedFunds_zscore_60d", "EWY_Korea_ret_20d", "TM_Telephone_vol_20d", "GILD_Gilead_ret_20d", "BTI_BritishAmerican_ret_5d", "XLY_Disc_vol_20d", "MS_MorganStanley_ret_1d", "MS_MorganStanley_ret_5d", "EMR_Emerson_ret_20d", "DOW_Price_zscore_60d", "US3M_Rate_zscore_60d", "CCI_CrownCastle_vol_20d", "EWG_Germany_ret_20d", "Core_PCE_zscore_60d", "NVDA_vol_20d", "SBUX_vol_20d", "AMD_ret_5d", "SJM_JM_Smucker_ret_5d", "BDX_Becton_Dickinson_ret_20d", "LUV_SouthwestAir_ret_5d", "Michigan_Sentiment_ret_20d", "US1Y_Rate_ret_5d", "MO_AltriaMG_ret_1d"], "is_new": true}, {"model_id": "new_h1_GLOBAL_RandomForest_N30_t0", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 1, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "MSTR_Bitcoin3_ret_5d", "EWM_Malaysia_zscore_60d", "MS_MorganStanley_ret_5d", "EFFR_ret_1d", "CI_Cigna_vol_20d", "US3Y_Rate_ret_5d", "SO_SouthernCo_ret_5d", "spx_vol_5d", "LOW_Lowes_ret_5d", "heston_var_ev_h3", "BLK_BlackRock_zscore_60d", "HangSeng_HK_ret_5d", "US6M_Rate_ret_20d", "HD_ret_5d", "EXC_Exelon_ret_1d", "HangSeng_HK_ret_1d", "ASX_Australia_vol_20d", "ORCL_zscore_60d", "Nikkei_Japan_zscore_60d", "heston_var_ev_h5", "EWC_Canada_zscore_60d", "HangSeng_HK_vol_20d", "EWA_Australia_ret_1d", "EWM_Malaysia_ret_1d", "US3M_Rate_zscore_60d", "NOC_Northrop_ret_20d", "LLY_zscore_60d", "AMD_ret_1d"], "is_new": true}, {"model_id": "new_h1_GLOBAL_RandomForest_N30_t1", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 1, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWA_Australia_ret_1d", "DE_Deere_ret_5d", "EOG_EOGResources_ret_5d", "MSTR_Bitcoin3_ret_5d", "DE_Deere_vol_20d", "ORCL_vol_20d", "Core_PCE_zscore_60d", "CLX_Clorox_vol_20d", "HUM_Humana_ret_5d", "DIS_vol_20d", "EXC_Exelon_ret_1d", "EWL_Switzerland_vol_20d", "NFCI_ret_5d", "DHR_ret_1d", "JNJ_ret_1d", "US1Y_Rate_ret_5d", "ITT_ITTInc_ret_5d", "GD_GeneralDynamics_zscore_60d", "DOW_Price_zscore_60d", "CPB_CampbellSoup_ret_5d", "SO_SouthernCo_ret_5d", "VRP_ma5", "AVB_AvalonBay_zscore_60d", "TGT_Target_zscore_60d", "SCHW_Schwab_ret_5d", "VVIX_ret_20d", "NEE_NextEra_ret_20d", "heston_ev_h3"], "is_new": true}, {"model_id": "new_h1_GLOBAL_RandomForest_N30_t2", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 1, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "spx_momentum_3d", "WTI_Oil_FRED_zscore_60d", "US3Y_Rate_ret_5d", "MSTR_Bitcoin3_ret_20d", "FedFunds_zscore_60d", "EWL_Switzerland_zscore_60d", "EWG_Germany_ret_20d", "US30Y_Rate_ret_20d", "Industrial_Production_zscore_60d", "EWQ_France_zscore_60d", "DE_Deere_vol_20d", "HD_ret_20d", "PFE_ret_1d", "ES_Evergy_ret_1d", "LUV_SouthwestAir_ret_5d", "BDX_Becton_Dickinson_ret_20d", "XLV_Health_zscore_60d", "TGT_Target_zscore_60d", "EWQ_France_ret_20d", "EWM_Malaysia_ret_1d", "EWS_Singapore_ret_5d", "EXC_Exelon_zscore_60d", "heston_var_ev_h7", "ASX_Australia_ret_5d", "heston_ev_h3", "Brent_Oil_FRED_ret_5d", "CCI_CrownCastle_vol_20d", "XLF_Fin_vol_20d"], "is_new": true}, {"model_id": "new_h1_GLOBAL_RandomForest_N30_t3", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 1, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "MRK_Merck_zscore_60d", "GD_GeneralDynamics_zscore_60d", "MO_AltriaMG_ret_1d", "PLD_Prologis_ret_5d", "MS_MorganStanley_ret_1d", "US3M_Rate_vol_20d", "vix_acceleration_1d", "SJM_JM_Smucker_ret_1d", "IYM_BasicMaterials_ret_20d", "PCAR_PaccarInc_ret_5d", "M_Macys_vol_20d", "VOD_Vodafone_zscore_60d", "heston_var_ev_h7", "IYR_US_REIT2_zscore_60d", "EWG_Germany_ret_20d", "LOW_Lowes_ret_5d", "EWQ_France_zscore_60d", "ORCL_vol_20d", "spx_vol_5d", "hmm_p_stress", "PPL_PPL_ret_1d", "GE_ret_1d", "MS_MorganStanley_ret_5d", "MSTR_Bitcoin3_ret_5d", "SJM_JM_Smucker_ret_5d", "MSTR_Bitcoin3_ret_20d", "BTI_BritishAmerican_ret_20d", "Nikkei_Japan_zscore_60d"], "is_new": true}, {"model_id": "new_h1_GLOBAL_RandomForest_N30_t4", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 1, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EQIX_Equinix_ret_5d", "JNJ_ret_1d", "HD_ret_5d", "MO_AltriaMG_ret_1d", "AMT_AmericanTower_ret_1d", "TM_Telephone_ret_1d", "AMD_ret_1d", "US3Y_Rate_ret_5d", "IYR_US_REIT2_zscore_60d", "XLY_Disc_vol_20d", "HangSeng_HK_vol_20d", "SLB_Schlumberger_ret_5d", "CLX_Clorox_vol_20d", "vix_acceleration_1d", "ES_Evergy_ret_1d", "M_Macys_vol_20d", "VVIX_ret_20d", "DHR_vol_20d", "ITT_ITTInc_ret_5d", "MRK_Merck_zscore_60d", "US5Y_Rate_ret_5d", "MSTR_Bitcoin3_ret_20d", "MS_MorganStanley_zscore_60d", "HangSeng_HK_ret_5d", "PAYX_Paychex_vol_20d", "EQR_Equity_ret_1d", "CTAS_Cintas_vol_20d", "EWQ_France_zscore_60d"], "is_new": true}, {"model_id": "new_h1_GLOBAL_RandomForest_N30_t5", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 1, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWQ_France_zscore_60d", "MO_AltriaMG_ret_1d", "VOD_Vodafone_zscore_60d", "Core_PCE_zscore_60d", "GD_GeneralDynamics_zscore_60d", "EQR_Equity_ret_1d", "MSTR_Bitcoin3_ret_5d", "DAX_Germany_vol_20d", "HD_ret_5d", "BA_ret_1d", "FedFunds_zscore_60d", "TXN_vol_20d", "US7Y_Rate_ret_20d", "EWC_Canada_zscore_60d", "EWY_Korea_ret_20d", "EFFR_vol_20d", "NOC_Northrop_ret_20d", "Core_CPI_zscore_60d", "HangSeng_HK_ret_5d", "INTC_ret_5d", "IBEX_Spain_ret_20d", "EWJ_Japan_vol_20d", "BDX_Becton_Dickinson_ret_20d", "PAYX_Paychex_zscore_60d", "US3M_Rate_zscore_60d", "NFCI_ret_5d", "EWL_Switzerland_zscore_60d", "PFE_ret_1d"], "is_new": true}, {"model_id": "new_h1_GLOBAL_RandomForest_N30_t6", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 1, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "TGT_Target_zscore_60d", "heston_var_ev_h7", "DAX_Germany_vol_20d", "SCHW_Schwab_ret_5d", "TED_Spread_zscore_60d", "LOW_Lowes_ret_20d", "ENB_EnbridgeInc_ret_1d", "DHR_vol_20d", "Core_PCE_zscore_60d", "CCI_CrownCastle_vol_20d", "MO_AltriaMG_ret_1d", "T_ret_1d", "PAYX_Paychex_vol_20d", "CMCSA_ret_1d", "US1Y_Rate_ret_5d", "HD_ret_1d", "INTC_ret_5d", "EFFR_vol_20d", "EWH_HongKong_ret_5d", "LOW_Lowes_ret_5d", "XLK_Tech_zscore_60d", "EQIX_Equinix_ret_5d", "Nikkei_Japan_zscore_60d", "SLB_Schlumberger_ret_1d", "PFE_ret_1d", "EXC_Exelon_zscore_60d", "GD_GeneralDynamics_zscore_60d", "SJM_JM_Smucker_ret_1d"], "is_new": true}, {"model_id": "new_h1_GLOBAL_RandomForest_N30_t7", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 1, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "heston_var_ev_h5", "EMR_Emerson_ret_20d", "heston_var_ev_h7", "AMZN_ret_5d", "TED_Spread_zscore_60d", "NFCI_ret_5d", "JNJ_ret_1d", "DIS_vol_20d", "EWL_Switzerland_zscore_60d", "PAYX_Paychex_ret_20d", "PG_ret_20d", "PAYX_Paychex_zscore_60d", "DE_Deere_ret_5d", "US5Y_Rate_ret_5d", "Nikkei_Japan_zscore_60d", "M_Macys_vol_20d", "ITT_ITTInc_ret_5d", "NEE_NextEra_ret_20d", "EWA_Australia_ret_1d", "EWA_Australia_zscore_60d", "PLD_Prologis_ret_5d", "3M_ret_5d", "CTAS_Cintas_vol_20d", "HD_ret_1d", "BLK_BlackRock_zscore_60d", "AXP_Amex_ret_20d", "LOW_Lowes_ret_20d", "EFFR_ret_1d"], "is_new": true}, {"model_id": "new_h1_GLOBAL_LogisticRegression_N5_t0", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 1, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "IWM_SmallCap_vol_20d", "hmm_p_stress", "SPY_zscore_60d"], "is_new": true}, {"model_id": "new_h1_GLOBAL_LogisticRegression_N5_t1", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 1, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "DOW_Price_zscore_60d", "SBUX_ret_5d", "EQIX_Equinix_ret_5d"], "is_new": true}, {"model_id": "new_h1_GLOBAL_LogisticRegression_N5_t2", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 1, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "ES_Evergy_ret_1d", "INTC_ret_5d", "PG_ret_20d"], "is_new": true}, {"model_id": "new_h1_GLOBAL_LogisticRegression_N5_t3", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 1, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "AMZN_ret_5d", "XLB_Materials_zscore_60d", "HangSeng_HK_ret_5d"], "is_new": true}, {"model_id": "new_h1_GLOBAL_LogisticRegression_N5_t4", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 1, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "TED_Spread_zscore_60d", "XOM_ret_20d", "CLX_Clorox_vol_20d"], "is_new": true}, {"model_id": "new_h1_GLOBAL_LogisticRegression_N5_t5", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 1, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "DAX_Germany_vol_20d", "EWC_Canada_zscore_60d", "MSTR_Bitcoin3_ret_1d"], "is_new": true}, {"model_id": "new_h1_GLOBAL_LogisticRegression_N5_t6", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 1, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "SPY_zscore_60d", "MS_MorganStanley_ret_5d", "EQIX_Equinix_ret_5d"], "is_new": true}, {"model_id": "new_h1_GLOBAL_LogisticRegression_N5_t7", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 1, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWY_Korea_ret_20d", "LLY_zscore_60d", "DAX_Germany_zscore_60d"], "is_new": true}, {"model_id": "new_h1_GLOBAL_LogisticRegression_N8_t0", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 1, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWQ_France_ret_20d", "DIS_vol_20d", "EOG_EOGResources_ret_5d", "DOW_Price_zscore_60d", "heston_var_ev_h3", "PAYX_Paychex_ret_20d"], "is_new": true}, {"model_id": "new_h1_GLOBAL_LogisticRegression_N8_t1", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 1, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "HD_ret_5d", "LUV_SouthwestAir_ret_5d", "QQQ_vol_20d", "EWL_Switzerland_zscore_60d", "LMT_LockheedMartin_vol_20d", "EWM_Malaysia_ret_1d"], "is_new": true}, {"model_id": "new_h1_GLOBAL_LogisticRegression_N8_t2", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 1, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "CLX_Clorox_vol_20d", "SLB_Schlumberger_ret_1d", "LMT_LockheedMartin_ret_1d", "ASX_Australia_ret_5d", "PPL_PPL_ret_1d", "MS_MorganStanley_zscore_60d"], "is_new": true}, {"model_id": "new_h1_GLOBAL_LogisticRegression_N8_t3", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 1, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "SLB_Schlumberger_ret_5d", "ES_Evergy_ret_1d", "MS_MorganStanley_ret_1d", "XLF_Fin_vol_20d", "SJM_JM_Smucker_ret_1d", "Core_PCE_zscore_60d"], "is_new": true}, {"model_id": "new_h1_GLOBAL_LogisticRegression_N8_t4", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 1, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "ASX_Australia_vol_20d", "CPB_CampbellSoup_ret_5d", "AMD_ret_5d", "SBUX_zscore_60d", "DAX_Germany_vol_20d", "EFFR_ret_1d"], "is_new": true}, {"model_id": "new_h1_GLOBAL_LogisticRegression_N8_t5", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 1, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "US1Y_Rate_ret_5d", "US30Y_Rate_ret_20d", "gjr_condvar_h1", "EQR_Equity_ret_1d", "CLX_Clorox_vol_20d", "SCHW_Schwab_ret_5d"], "is_new": true}, {"model_id": "new_h1_GLOBAL_LogisticRegression_N8_t6", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 1, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "Michigan_Sentiment_ret_20d", "Retail_Sales_zscore_60d", "EWG_Germany_ret_20d", "IWM_SmallCap_vol_20d", "PAYX_Paychex_ret_20d", "FedFunds_zscore_60d"], "is_new": true}, {"model_id": "new_h1_GLOBAL_LogisticRegression_N8_t7", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 1, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EFFR_ret_1d", "DHR_vol_20d", "EMR_Emerson_ret_20d", "SLB_Schlumberger_ret_5d", "AMGN_Amgen_ret_1d", "heston_ev_h3"], "is_new": true}, {"model_id": "new_h1_GLOBAL_LogisticRegression_N10_t0", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 1, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "LLY_zscore_60d", "US30Y_Rate_ret_20d", "vix_mean_abs_ret_5d", "US3M_Rate_vol_20d", "EFFR_ret_1d", "vix_acceleration_1d", "gjr_condvar_h1", "ES_Evergy_ret_1d"], "is_new": true}, {"model_id": "new_h1_GLOBAL_LogisticRegression_N10_t1", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 1, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "MSTR_Bitcoin3_ret_1d", "EQR_Equity_ret_1d", "XLY_Disc_vol_20d", "Industrial_Production_zscore_60d", "EWJ_Japan_vol_20d", "MRK_Merck_zscore_60d", "spx_abs_ret_max_5d", "AXP_Amex_vol_20d"], "is_new": true}, {"model_id": "new_h1_GLOBAL_LogisticRegression_N10_t2", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 1, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "BTI_BritishAmerican_ret_5d", "heston_var_ev_h7", "Core_CPI_zscore_60d", "TGT_Target_zscore_60d", "BDX_Becton_Dickinson_ret_20d", "GE_ret_1d", "MS_MorganStanley_zscore_60d", "SBUX_zscore_60d"], "is_new": true}, {"model_id": "new_h1_GLOBAL_LogisticRegression_N10_t3", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 1, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "SBUX_vol_20d", "vix_mean_abs_ret_5d", "INTC_ret_1d", "EWM_Malaysia_vol_20d", "US30Y_Rate_ret_20d", "HangSeng_HK_vol_20d", "DAX_Germany_vol_20d", "spx_abs_ret_max_5d"], "is_new": true}, {"model_id": "new_h1_GLOBAL_LogisticRegression_N10_t4", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 1, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "spx_momentum_3d", "Brent_Oil_FRED_ret_20d", "EWL_Switzerland_zscore_60d", "MSTR_Bitcoin3_ret_5d", "EOG_EOGResources_vol_20d", "M_Macys_vol_20d", "spx_vol_5d", "BDX_Becton_Dickinson_ret_20d"], "is_new": true}, {"model_id": "new_h1_GLOBAL_LogisticRegression_N10_t5", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 1, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "US3Y_Rate_ret_5d", "QQQ_vol_20d", "spx_momentum_3d", "LUV_SouthwestAir_ret_5d", "MSTR_Bitcoin3_ret_5d", "BTI_BritishAmerican_ret_20d", "INTC_ret_1d", "GILD_Gilead_ret_20d"], "is_new": true}, {"model_id": "new_h1_GLOBAL_LogisticRegression_N10_t6", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 1, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWY_Korea_zscore_60d", "US3M_Rate_zscore_60d", "Brent_Oil_FRED_ret_20d", "EOG_EOGResources_vol_20d", "XOM_ret_20d", "spx_momentum_3d", "BTI_BritishAmerican_ret_5d", "MO_AltriaMG_ret_1d"], "is_new": true}, {"model_id": "new_h1_GLOBAL_LogisticRegression_N10_t7", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 1, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "Brent_Oil_FRED_ret_5d", "US3M_Rate_vol_20d", "JNJ_ret_1d", "IYM_BasicMaterials_ret_20d", "Nikkei_Japan_vol_20d", "GE_ret_1d", "AMGN_Amgen_ret_1d", "US3M_Rate_zscore_60d"], "is_new": true}, {"model_id": "new_h1_GLOBAL_LogisticRegression_N12_t0", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 1, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "VRP_ma5", "ITT_ITTInc_ret_5d", "PAYX_Paychex_zscore_60d", "T_ret_1d", "SJM_JM_Smucker_ret_1d", "LOW_Lowes_ret_20d", "ES_Evergy_ret_1d", "BLK_BlackRock_zscore_60d", "Retail_Sales_zscore_60d", "3M_ret_5d"], "is_new": true}, {"model_id": "new_h1_GLOBAL_LogisticRegression_N12_t1", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 1, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "ORCL_vol_20d", "EFFR_vol_20d", "vix_acceleration_1d", "AMD_ret_5d", "Brent_Oil_FRED_ret_20d", "XOM_ret_1d", "hmm_p_stress", "SJM_JM_Smucker_ret_1d", "CPB_CampbellSoup_ret_20d", "DE_Deere_vol_20d"], "is_new": true}, {"model_id": "new_h1_GLOBAL_LogisticRegression_N12_t2", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 1, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "Brent_Oil_FRED_ret_20d", "EWG_Germany_ret_20d", "US3Y_Rate_ret_5d", "EWS_Singapore_ret_5d", "Nikkei_Japan_zscore_60d", "XLY_Disc_vol_20d", "Michigan_Sentiment_ret_20d", "PG_ret_20d", "TGT_Target_zscore_60d", "US3M_Rate_zscore_60d"], "is_new": true}, {"model_id": "new_h1_GLOBAL_LogisticRegression_N12_t3", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 1, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "US3M_Rate_vol_20d", "T10Y2Y_Spread_ret_5d", "QQQ_vol_20d", "EWL_Switzerland_vol_20d", "Brent_Oil_FRED_ret_20d", "ENB_EnbridgeInc_ret_1d", "Nikkei_Japan_vol_20d", "SBUX_zscore_60d", "SBUX_ret_5d", "EWM_Malaysia_zscore_60d"], "is_new": true}, {"model_id": "new_h1_GLOBAL_LogisticRegression_N12_t4", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 1, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "hmm_p_stress", "MS_MorganStanley_ret_1d", "AORD_AUS_zscore_60d", "PFE_ret_1d", "Industrial_Production_zscore_60d", "US5Y_Rate_ret_5d", "TED_Spread_zscore_60d", "TGT_Target_zscore_60d", "XOM_ret_1d", "QQQ_vol_20d"], "is_new": true}, {"model_id": "new_h1_GLOBAL_LogisticRegression_N12_t5", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 1, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "heston_var_ev_h3", "EWH_HongKong_ret_5d", "gjr_condvar_h1", "US3Y_Rate_ret_5d", "IBEX_Spain_ret_20d", "EWG_Germany_ret_20d", "Core_CPI_zscore_60d", "XLB_Materials_zscore_60d", "CMCSA_ret_1d", "MSTR_Bitcoin3_ret_20d"], "is_new": true}, {"model_id": "new_h1_GLOBAL_LogisticRegression_N12_t6", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 1, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "SO_SouthernCo_ret_5d", "US6M_Rate_ret_20d", "heston_var_ev_h3", "IYR_US_REIT2_zscore_60d", "EQR_Equity_ret_1d", "EWM_Malaysia_zscore_60d", "DAX_Germany_vol_20d", "DIS_vol_20d", "AXP_Amex_ret_20d", "EFFR_vol_20d"], "is_new": true}, {"model_id": "new_h1_GLOBAL_LogisticRegression_N12_t7", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 1, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "LMT_LockheedMartin_ret_1d", "SBUX_vol_20d", "MSTR_Bitcoin3_ret_1d", "GE_ret_1d", "HD_ret_20d", "US3M_Rate_vol_20d", "heston_var_ev_h3", "PCAR_PaccarInc_ret_5d", "Retail_Sales_zscore_60d", "3M_vol_20d"], "is_new": true}, {"model_id": "new_h1_GLOBAL_LogisticRegression_N15_t0", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 1, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "NVDA_vol_20d", "US30Y_Rate_ret_20d", "PG_ret_20d", "MS_MorganStanley_zscore_60d", "HangSeng_HK_vol_20d", "IBEX_Spain_ret_20d", "TXN_vol_20d", "vix_acceleration_1d", "ORCL_vol_20d", "SJM_JM_Smucker_ret_1d", "M_Macys_vol_20d", "HangSeng_HK_ret_1d", "US3Y_Rate_ret_5d"], "is_new": true}, {"model_id": "new_h1_GLOBAL_LogisticRegression_N15_t1", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 1, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "NVDA_vol_20d", "EWG_Germany_ret_20d", "MSTR_Bitcoin3_ret_5d", "LLY_zscore_60d", "Nikkei_Japan_vol_20d", "ASX_Australia_ret_5d", "EXC_Exelon_ret_1d", "IYM_BasicMaterials_ret_20d", "AMT_AmericanTower_ret_1d", "NEE_NextEra_ret_20d", "EWS_Singapore_ret_5d", "LMT_LockheedMartin_vol_20d", "INTC_ret_1d"], "is_new": true}, {"model_id": "new_h1_GLOBAL_LogisticRegression_N15_t2", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 1, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "ASX_Australia_vol_20d", "VOD_Vodafone_zscore_60d", "MSTR_Bitcoin3_ret_5d", "GE_ret_1d", "US3M_Rate_vol_20d", "Nikkei_Japan_vol_20d", "XLV_Health_zscore_60d", "ES_Evergy_ret_1d", "PG_ret_20d", "ORCL_zscore_60d", "CCI_CrownCastle_vol_20d", "CTAS_Cintas_vol_20d", "heston_ev_h3"], "is_new": true}, {"model_id": "new_h1_GLOBAL_LogisticRegression_N15_t3", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 1, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWM_Malaysia_ret_1d", "ITT_ITTInc_ret_5d", "EWG_Germany_vol_20d", "AXP_Amex_vol_20d", "MO_AltriaMG_ret_1d", "VRP_ma5", "PLD_Prologis_ret_5d", "AMD_ret_1d", "EQIX_Equinix_ret_5d", "LMT_LockheedMartin_ret_1d", "CTAS_Cintas_vol_20d", "EMR_Emerson_ret_20d", "TED_Spread_zscore_60d"], "is_new": true}, {"model_id": "new_h1_GLOBAL_LogisticRegression_N15_t4", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 1, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "Nikkei_Japan_vol_20d", "HangSeng_HK_ret_1d", "EWQ_France_ret_20d", "CCI_CrownCastle_vol_20d", "INTC_ret_1d", "XLV_Health_zscore_60d", "HD_zscore_60d", "TED_Spread_vol_20d", "XLY_Disc_vol_20d", "US30Y_Rate_ret_20d", "US3M_Rate_zscore_60d", "IYR_US_REIT2_zscore_60d", "XLK_Tech_zscore_60d"], "is_new": true}, {"model_id": "new_h1_GLOBAL_LogisticRegression_N15_t5", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 1, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "spx_momentum_3d", "ORCL_vol_20d", "EWY_Korea_ret_20d", "HangSeng_HK_ret_1d", "IYM_BasicMaterials_ret_20d", "FedFunds_zscore_60d", "HD_zscore_60d", "EWL_Switzerland_zscore_60d", "HangSeng_HK_vol_20d", "EXC_Exelon_ret_1d", "Brent_Oil_FRED_ret_20d", "XLK_Tech_zscore_60d", "DAX_Germany_vol_20d"], "is_new": true}, {"model_id": "new_h1_GLOBAL_LogisticRegression_N15_t6", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 1, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "GILD_Gilead_ret_20d", "US6M_Rate_ret_20d", "PAYX_Paychex_zscore_60d", "HD_ret_20d", "EWM_Malaysia_ret_1d", "US5Y_Rate_ret_5d", "GE_ret_1d", "NEE_NextEra_ret_20d", "FedFunds_zscore_60d", "AMGN_Amgen_ret_1d", "TED_Spread_vol_20d", "spx_momentum_3d", "US30Y_Rate_ret_20d"], "is_new": true}, {"model_id": "new_h1_GLOBAL_LogisticRegression_N15_t7", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 1, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "VRP_ma5", "IYM_BasicMaterials_ret_20d", "SJM_JM_Smucker_ret_5d", "NVDA_vol_20d", "DAX_Germany_zscore_60d", "XLB_Materials_zscore_60d", "Retail_Sales_zscore_60d", "EWL_Switzerland_vol_20d", "US5Y_Rate_ret_5d", "vix_mean_abs_ret_5d", "TGT_Target_zscore_60d", "heston_var_ev_h5", "T10Y2Y_Spread_ret_5d"], "is_new": true}, {"model_id": "new_h1_GLOBAL_LogisticRegression_N20_t0", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 1, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EFFR_vol_20d", "CPB_CampbellSoup_vol_20d", "AMD_ret_1d", "US3Y_Rate_ret_5d", "PAYX_Paychex_vol_20d", "DE_Deere_ret_5d", "BLK_BlackRock_zscore_60d", "EOG_EOGResources_vol_20d", "Core_CPI_zscore_60d", "Retail_Sales_zscore_60d", "AXP_Amex_ret_20d", "IWM_SmallCap_vol_20d", "HangSeng_HK_ret_5d", "SPY_zscore_60d", "LMT_LockheedMartin_ret_1d", "EWA_Australia_zscore_60d", "HD_ret_5d", "gjr_condvar_h1"], "is_new": true}, {"model_id": "new_h1_GLOBAL_LogisticRegression_N20_t1", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 1, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "PPL_PPL_ret_1d", "US7Y_Rate_ret_20d", "DE_Deere_ret_5d", "DAX_Germany_zscore_60d", "ASX_Australia_vol_20d", "EWS_Singapore_ret_5d", "AMD_ret_1d", "AVB_AvalonBay_zscore_60d", "HangSeng_HK_ret_5d", "HD_ret_1d", "US3M_Rate_zscore_60d", "Nikkei_Japan_zscore_60d", "MSTR_Bitcoin3_ret_20d", "BTI_BritishAmerican_ret_20d", "DIS_vol_20d", "XLK_Tech_zscore_60d", "EWM_Malaysia_ret_1d", "BA_ret_1d"], "is_new": true}, {"model_id": "new_h1_GLOBAL_LogisticRegression_N20_t2", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 1, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "DE_Deere_vol_20d", "Core_CPI_zscore_60d", "EWG_Germany_vol_20d", "PG_ret_20d", "IYR_US_REIT2_zscore_60d", "MS_MorganStanley_ret_5d", "LOW_Lowes_ret_20d", "TED_Spread_zscore_60d", "VVIX_ret_20d", "EWA_Australia_ret_1d", "IYM_BasicMaterials_ret_20d", "US7Y_Rate_ret_20d", "Industrial_Production_zscore_60d", "XOM_ret_20d", "PPL_PPL_ret_1d", "3M_ret_5d", "US1Y_Rate_ret_5d", "Nikkei_Japan_zscore_60d"], "is_new": true}, {"model_id": "new_h1_GLOBAL_LogisticRegression_N20_t3", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 1, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "LOW_Lowes_ret_5d", "US1Y_Rate_ret_20d", "HangSeng_HK_ret_1d", "EWA_Australia_ret_1d", "EXC_Exelon_zscore_60d", "IWM_SmallCap_vol_20d", "MS_MorganStanley_zscore_60d", "MS_MorganStanley_ret_1d", "EWL_Switzerland_zscore_60d", "CI_Cigna_vol_20d", "Brent_Oil_FRED_ret_20d", "EWM_Malaysia_ret_1d", "TM_Telephone_vol_20d", "T_ret_1d", "MO_AltriaMG_ret_1d", "AXP_Amex_vol_20d", "EWG_Germany_vol_20d", "HUM_Humana_ret_5d"], "is_new": true}, {"model_id": "new_h1_GLOBAL_LogisticRegression_N20_t4", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 1, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "vix_mean_abs_ret_5d", "DE_Deere_vol_20d", "ASX_Australia_ret_5d", "EWY_Korea_ret_20d", "EWA_Australia_zscore_60d", "IBEX_Spain_ret_20d", "PFE_ret_1d", "HD_zscore_60d", "EQR_Equity_ret_1d", "FedFunds_zscore_60d", "NEE_NextEra_ret_20d", "EWH_HongKong_ret_5d", "spx_abs_ret_max_5d", "CPB_CampbellSoup_zscore_60d", "AMGN_Amgen_ret_1d", "HangSeng_HK_ret_5d", "EWM_Malaysia_zscore_60d", "spx_vol_5d"], "is_new": true}, {"model_id": "new_h1_GLOBAL_LogisticRegression_N20_t5", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 1, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "ASX_Australia_ret_5d", "US1Y_Rate_ret_20d", "hmm_p_stress", "INTC_ret_1d", "DHR_vol_20d", "US3M_Rate_zscore_60d", "HD_ret_1d", "MSTR_Bitcoin3_ret_20d", "US7Y_Rate_ret_20d", "AXP_Amex_vol_20d", "EFFR_ret_1d", "spx_abs_ret_max_5d", "CLX_Clorox_vol_20d", "heston_var_ev_h7", "spx_vol_5d", "SBUX_vol_20d", "ORCL_zscore_60d", "HD_ret_5d"], "is_new": true}, {"model_id": "new_h1_GLOBAL_LogisticRegression_N20_t6", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 1, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "ASX_Australia_vol_20d", "HangSeng_HK_ret_5d", "EWH_HongKong_ret_5d", "INTC_ret_5d", "BDX_Becton_Dickinson_ret_20d", "EFFR_ret_1d", "EWM_Malaysia_vol_20d", "TM_Telephone_ret_1d", "LUV_SouthwestAir_ret_5d", "AXP_Amex_vol_20d", "EMR_Emerson_ret_20d", "US30Y_Rate_ret_20d", "XLK_Tech_zscore_60d", "HD_ret_1d", "NWL_Newell_ret_20d", "MS_MorganStanley_ret_5d", "Industrial_Production_zscore_60d", "EQR_Equity_ret_1d"], "is_new": true}, {"model_id": "new_h1_GLOBAL_LogisticRegression_N20_t7", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 1, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWG_Germany_ret_20d", "DE_Deere_vol_20d", "T10Y2Y_Spread_ret_5d", "EFFR_ret_1d", "XOM_ret_1d", "EWM_Malaysia_zscore_60d", "AMD_ret_1d", "AMGN_Amgen_ret_1d", "IYM_BasicMaterials_ret_20d", "M_Macys_vol_20d", "US3Y_Rate_ret_5d", "CPB_CampbellSoup_zscore_60d", "EWL_Switzerland_vol_20d", "EWG_Germany_vol_20d", "EWM_Malaysia_vol_20d", "AMT_AmericanTower_ret_1d", "EWL_Switzerland_zscore_60d", "EWC_Canada_zscore_60d"], "is_new": true}, {"model_id": "new_h1_GLOBAL_LogisticRegression_N25_t0", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 1, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "BTI_BritishAmerican_ret_5d", "DAX_Germany_vol_20d", "Nikkei_Japan_vol_20d", "PCAR_PaccarInc_ret_5d", "LUV_SouthwestAir_ret_5d", "AMD_ret_1d", "ORCL_zscore_60d", "spx_vol_5d", "DHR_vol_20d", "Retail_Sales_zscore_60d", "XLK_Tech_zscore_60d", "gjr_condvar_h1", "LLY_zscore_60d", "INTC_ret_1d", "EWY_Korea_ret_20d", "BDX_Becton_Dickinson_ret_20d", "EOG_EOGResources_vol_20d", "HD_ret_1d", "MO_AltriaMG_ret_1d", "EWH_HongKong_ret_5d", "XOM_ret_20d", "CCI_CrownCastle_vol_20d", "XLF_Fin_vol_20d"], "is_new": true}, {"model_id": "new_h1_GLOBAL_LogisticRegression_N25_t1", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 1, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWQ_France_ret_20d", "AORD_AUS_zscore_60d", "EWL_Switzerland_vol_20d", "GE_ret_1d", "DE_Deere_ret_5d", "AXP_Amex_ret_20d", "DAX_Germany_zscore_60d", "TM_Telephone_vol_20d", "HUM_Humana_ret_5d", "PFE_ret_1d", "BTI_BritishAmerican_ret_5d", "US6M_Rate_ret_20d", "AMGN_Amgen_ret_1d", "CMCSA_ret_1d", "PCAR_PaccarInc_ret_5d", "EMR_Emerson_ret_20d", "XLK_Tech_zscore_60d", "spx_abs_ret_max_5d", "HD_zscore_60d", "BA_ret_1d", "MRK_Merck_zscore_60d", "BDX_Becton_Dickinson_ret_20d", "ORCL_zscore_60d"], "is_new": true}, {"model_id": "new_h1_GLOBAL_LogisticRegression_N25_t2", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 1, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "CPB_CampbellSoup_vol_20d", "QQQ_vol_20d", "heston_var_ev_h5", "heston_ev_h3", "SCHW_Schwab_ret_5d", "ITT_ITTInc_ret_5d", "EXC_Exelon_zscore_60d", "NFCI_ret_5d", "CPB_CampbellSoup_zscore_60d", "XOM_ret_1d", "WTI_Oil_FRED_zscore_60d", "EQR_Equity_ret_1d", "FedFunds_zscore_60d", "AXP_Amex_vol_20d", "MSTR_Bitcoin3_ret_5d", "LOW_Lowes_ret_5d", "TM_Telephone_vol_20d", "EWM_Malaysia_vol_20d", "TM_Telephone_ret_1d", "ORCL_zscore_60d", "EFFR_ret_1d", "PPL_PPL_ret_1d", "DIS_vol_20d"], "is_new": true}, {"model_id": "new_h1_GLOBAL_LogisticRegression_N25_t3", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 1, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "INTC_ret_5d", "WTI_Oil_FRED_zscore_60d", "AVB_AvalonBay_zscore_60d", "US7Y_Rate_ret_20d", "JNJ_ret_1d", "MSTR_Bitcoin3_ret_5d", "MS_MorganStanley_ret_5d", "TGT_Target_zscore_60d", "HD_ret_20d", "EXC_Exelon_ret_1d", "SLB_Schlumberger_ret_1d", "DIS_vol_20d", "EWY_Korea_ret_20d", "SBUX_vol_20d", "XOM_ret_1d", "EWA_Australia_ret_1d", "PAYX_Paychex_zscore_60d", "US3Y_Rate_ret_5d", "Brent_Oil_FRED_ret_5d", "ES_Evergy_ret_1d", "CPB_CampbellSoup_ret_5d", "DE_Deere_ret_5d", "CI_Cigna_vol_20d"], "is_new": true}, {"model_id": "new_h1_GLOBAL_LogisticRegression_N25_t4", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 1, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "SO_SouthernCo_ret_5d", "AVB_AvalonBay_zscore_60d", "EWQ_France_ret_20d", "LLY_zscore_60d", "US1Y_Rate_ret_20d", "NOC_Northrop_ret_20d", "XLV_Health_zscore_60d", "AXP_Amex_vol_20d", "Brent_Oil_FRED_ret_20d", "PCAR_PaccarInc_ret_5d", "heston_var_ev_h5", "SBUX_vol_20d", "HD_zscore_60d", "ASX_Australia_vol_20d", "EWM_Malaysia_ret_1d", "WTI_Oil_FRED_zscore_60d", "DOW_Price_zscore_60d", "LMT_LockheedMartin_vol_20d", "CPB_CampbellSoup_vol_20d", "VOD_Vodafone_zscore_60d", "EWY_Korea_zscore_60d", "XLB_Materials_zscore_60d", "EQIX_Equinix_ret_5d"], "is_new": true}, {"model_id": "new_h1_GLOBAL_LogisticRegression_N25_t5", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 1, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "CPB_CampbellSoup_vol_20d", "EOG_EOGResources_vol_20d", "vix_acceleration_1d", "DHR_vol_20d", "SCHW_Schwab_ret_5d", "MS_MorganStanley_ret_1d", "EXC_Exelon_ret_1d", "EWY_Korea_ret_20d", "EWL_Switzerland_zscore_60d", "US1Y_Rate_ret_5d", "PCAR_PaccarInc_ret_5d", "EQR_Equity_ret_1d", "EMR_Emerson_ret_20d", "AMT_AmericanTower_ret_1d", "LOW_Lowes_ret_5d", "HD_zscore_60d", "DOW_Price_zscore_60d", "spx_momentum_3d", "CTAS_Cintas_vol_20d", "MSTR_Bitcoin3_ret_1d", "EQIX_Equinix_ret_5d", "VVIX_ret_20d", "ITT_ITTInc_ret_5d"], "is_new": true}, {"model_id": "new_h1_GLOBAL_LogisticRegression_N25_t6", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 1, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWA_Australia_zscore_60d", "3M_ret_5d", "TM_Telephone_vol_20d", "EWC_Canada_zscore_60d", "EWG_Germany_ret_20d", "HD_zscore_60d", "HUM_Humana_ret_5d", "HangSeng_HK_ret_5d", "PAYX_Paychex_ret_20d", "AMGN_Amgen_ret_1d", "HangSeng_HK_vol_20d", "SJM_JM_Smucker_ret_5d", "SPY_zscore_60d", "SJM_JM_Smucker_ret_1d", "SLB_Schlumberger_ret_5d", "WTI_Oil_FRED_zscore_60d", "US6M_Rate_ret_20d", "IYM_BasicMaterials_ret_20d", "AMD_ret_5d", "GE_ret_1d", "PG_ret_20d", "AMT_AmericanTower_ret_1d", "ENB_EnbridgeInc_ret_1d"], "is_new": true}, {"model_id": "new_h1_GLOBAL_LogisticRegression_N25_t7", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 1, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "Brent_Oil_FRED_ret_20d", "XLK_Tech_zscore_60d", "INTC_ret_1d", "EWA_Australia_ret_1d", "PAYX_Paychex_vol_20d", "SJM_JM_Smucker_ret_1d", "EXC_Exelon_ret_1d", "XOM_ret_1d", "EQR_Equity_ret_1d", "MS_MorganStanley_zscore_60d", "Nikkei_Japan_vol_20d", "EWH_HongKong_ret_5d", "EFFR_vol_20d", "WTI_Oil_FRED_zscore_60d", "INTC_ret_5d", "NOC_Northrop_ret_20d", "EWM_Malaysia_zscore_60d", "GD_GeneralDynamics_zscore_60d", "ES_Evergy_ret_1d", "PAYX_Paychex_zscore_60d", "hmm_p_stress", "QQQ_vol_20d", "XLV_Health_zscore_60d"], "is_new": true}, {"model_id": "new_h1_GLOBAL_LogisticRegression_N30_t0", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 1, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "AMGN_Amgen_ret_1d", "NVDA_vol_20d", "CCI_CrownCastle_vol_20d", "M_Macys_vol_20d", "LMT_LockheedMartin_ret_1d", "TM_Telephone_vol_20d", "CI_Cigna_vol_20d", "DHR_vol_20d", "XLB_Materials_zscore_60d", "DE_Deere_vol_20d", "AORD_AUS_zscore_60d", "MO_AltriaMG_ret_1d", "US3Y_Rate_ret_5d", "EFFR_vol_20d", "DOW_Price_zscore_60d", "VRP_ma5", "HD_ret_5d", "Core_CPI_zscore_60d", "Brent_Oil_FRED_ret_20d", "ITT_ITTInc_ret_5d", "NFCI_ret_5d", "EWL_Switzerland_vol_20d", "AVB_AvalonBay_zscore_60d", "XLV_Health_zscore_60d", "INTC_ret_5d", "INTC_ret_1d", "spx_momentum_3d", "EWM_Malaysia_zscore_60d"], "is_new": true}, {"model_id": "new_h1_GLOBAL_LogisticRegression_N30_t1", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 1, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "heston_var_ev_h7", "NEE_NextEra_ret_20d", "EOG_EOGResources_vol_20d", "XOM_ret_1d", "HUM_Humana_ret_5d", "EQR_Equity_ret_1d", "AVB_AvalonBay_zscore_60d", "Nikkei_Japan_zscore_60d", "MSTR_Bitcoin3_ret_1d", "Brent_Oil_FRED_ret_5d", "IYR_US_REIT2_zscore_60d", "EXC_Exelon_zscore_60d", "spx_momentum_3d", "SO_SouthernCo_ret_5d", "AMZN_ret_5d", "US1Y_Rate_ret_20d", "Industrial_Production_zscore_60d", "CPB_CampbellSoup_ret_20d", "EWA_Australia_zscore_60d", "EWG_Germany_vol_20d", "CPB_CampbellSoup_zscore_60d", "EWM_Malaysia_ret_1d", "TM_Telephone_ret_1d", "vix_mean_abs_ret_5d", "AXP_Amex_ret_20d", "GE_ret_1d", "TGT_Target_zscore_60d", "VOD_Vodafone_zscore_60d"], "is_new": true}, {"model_id": "new_h1_GLOBAL_LogisticRegression_N30_t2", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 1, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EQIX_Equinix_ret_5d", "DOW_Price_zscore_60d", "CPB_CampbellSoup_ret_20d", "EOG_EOGResources_vol_20d", "Brent_Oil_FRED_ret_5d", "MRK_Merck_zscore_60d", "HUM_Humana_ret_5d", "SLB_Schlumberger_ret_5d", "CTAS_Cintas_vol_20d", "hmm_p_stress", "TM_Telephone_ret_1d", "EFFR_ret_1d", "ORCL_vol_20d", "TXN_vol_20d", "NOC_Northrop_ret_20d", "gjr_condvar_h1", "SJM_JM_Smucker_ret_1d", "M_Macys_vol_20d", "HangSeng_HK_ret_1d", "AMD_ret_1d", "CPB_CampbellSoup_vol_20d", "US7Y_Rate_ret_20d", "heston_var_ev_h3", "TED_Spread_zscore_60d", "IWM_SmallCap_vol_20d", "FedFunds_zscore_60d", "US1Y_Rate_ret_5d", "XLK_Tech_zscore_60d"], "is_new": true}, {"model_id": "new_h1_GLOBAL_LogisticRegression_N30_t3", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 1, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "DE_Deere_vol_20d", "3M_ret_5d", "AVB_AvalonBay_zscore_60d", "MS_MorganStanley_ret_1d", "EQIX_Equinix_ret_5d", "US3M_Rate_zscore_60d", "SJM_JM_Smucker_ret_5d", "SJM_JM_Smucker_ret_1d", "PAYX_Paychex_ret_20d", "TED_Spread_vol_20d", "Retail_Sales_zscore_60d", "EWS_Singapore_ret_5d", "INTC_ret_1d", "T10Y2Y_Spread_ret_5d", "IWM_SmallCap_vol_20d", "spx_momentum_3d", "M_Macys_vol_20d", "HD_ret_20d", "XLF_Fin_vol_20d", "IBEX_Spain_ret_20d", "Core_PCE_zscore_60d", "ORCL_vol_20d", "DE_Deere_ret_5d", "EWL_Switzerland_vol_20d", "HUM_Humana_ret_5d", "EQR_Equity_ret_1d", "CTAS_Cintas_vol_20d", "DHR_vol_20d"], "is_new": true}, {"model_id": "new_h1_GLOBAL_LogisticRegression_N30_t4", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 1, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "Brent_Oil_FRED_ret_5d", "EWA_Australia_zscore_60d", "EQR_Equity_ret_1d", "AMGN_Amgen_ret_1d", "HangSeng_HK_vol_20d", "BTI_BritishAmerican_ret_20d", "AXP_Amex_ret_20d", "HD_ret_20d", "EMR_Emerson_ret_20d", "XLK_Tech_zscore_60d", "EWY_Korea_zscore_60d", "TGT_Target_zscore_60d", "EWL_Switzerland_vol_20d", "CTAS_Cintas_vol_20d", "heston_var_ev_h5", "SBUX_vol_20d", "SLB_Schlumberger_ret_5d", "US3M_Rate_zscore_60d", "XLY_Disc_vol_20d", "CMCSA_ret_1d", "PPL_PPL_ret_1d", "PAYX_Paychex_vol_20d", "EFFR_ret_1d", "NEE_NextEra_ret_20d", "HD_ret_1d", "TED_Spread_vol_20d", "CPB_CampbellSoup_ret_20d", "heston_var_ev_h3"], "is_new": true}, {"model_id": "new_h1_GLOBAL_LogisticRegression_N30_t5", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 1, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "BDX_Becton_Dickinson_ret_20d", "EWL_Switzerland_zscore_60d", "Nikkei_Japan_zscore_60d", "NWL_Newell_ret_20d", "NEE_NextEra_ret_20d", "AMT_AmericanTower_ret_1d", "EOG_EOGResources_ret_5d", "3M_ret_5d", "BLK_BlackRock_zscore_60d", "3M_vol_20d", "LMT_LockheedMartin_ret_1d", "EWC_Canada_zscore_60d", "EMR_Emerson_ret_20d", "TM_Telephone_vol_20d", "Industrial_Production_zscore_60d", "EWA_Australia_zscore_60d", "TXN_vol_20d", "MS_MorganStanley_zscore_60d", "EWA_Australia_ret_1d", "CPB_CampbellSoup_ret_20d", "heston_var_ev_h5", "ITT_ITTInc_ret_5d", "AXP_Amex_ret_20d", "IWM_SmallCap_vol_20d", "EQIX_Equinix_ret_5d", "GE_ret_1d", "EWS_Singapore_ret_5d", "TED_Spread_zscore_60d"], "is_new": true}, {"model_id": "new_h1_GLOBAL_LogisticRegression_N30_t6", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 1, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "NOC_Northrop_ret_20d", "XLV_Health_zscore_60d", "EOG_EOGResources_ret_5d", "spx_momentum_3d", "US3M_Rate_vol_20d", "XLK_Tech_zscore_60d", "BLK_BlackRock_zscore_60d", "heston_var_ev_h7", "PAYX_Paychex_zscore_60d", "MS_MorganStanley_zscore_60d", "TGT_Target_zscore_60d", "Nikkei_Japan_zscore_60d", "EMR_Emerson_ret_20d", "PFE_ret_1d", "ORCL_vol_20d", "EWS_Singapore_ret_5d", "EWM_Malaysia_vol_20d", "EQIX_Equinix_ret_5d", "US1Y_Rate_ret_5d", "EWG_Germany_vol_20d", "SJM_JM_Smucker_ret_5d", "MSTR_Bitcoin3_ret_5d", "VVIX_ret_20d", "EWJ_Japan_vol_20d", "SBUX_ret_5d", "NEE_NextEra_ret_20d", "CI_Cigna_vol_20d", "SLB_Schlumberger_ret_1d"], "is_new": true}, {"model_id": "new_h1_GLOBAL_LogisticRegression_N30_t7", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 1, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "MS_MorganStanley_zscore_60d", "PCAR_PaccarInc_ret_5d", "CPB_CampbellSoup_vol_20d", "EWY_Korea_ret_20d", "ORCL_vol_20d", "US1Y_Rate_ret_20d", "HangSeng_HK_vol_20d", "CPB_CampbellSoup_zscore_60d", "3M_ret_5d", "Brent_Oil_FRED_ret_5d", "EWA_Australia_ret_1d", "EFFR_ret_1d", "SLB_Schlumberger_ret_1d", "EFFR_vol_20d", "EWG_Germany_ret_20d", "EWM_Malaysia_ret_1d", "XLF_Fin_vol_20d", "XOM_ret_20d", "CPB_CampbellSoup_ret_5d", "TM_Telephone_vol_20d", "SLB_Schlumberger_ret_5d", "XLK_Tech_zscore_60d", "LMT_LockheedMartin_ret_1d", "ES_Evergy_ret_1d", "PG_ret_20d", "BTI_BritishAmerican_ret_20d", "TED_Spread_zscore_60d", "Retail_Sales_zscore_60d"], "is_new": true}, {"model_id": "new_h2_CALM_XGBoost_N5_t0", "algo": "XGBoost", "regime": "CALM", "horizon": 2, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "SCHW_Schwab_ret_5d", "MSTR_Bitcoin3_ret_5d", "CPB_CampbellSoup_ret_20d"], "is_new": true}, {"model_id": "new_h2_CALM_XGBoost_N5_t1", "algo": "XGBoost", "regime": "CALM", "horizon": 2, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "US6M_Rate_ret_20d", "ENB_EnbridgeInc_ret_1d", "3M_vol_20d"], "is_new": true}, {"model_id": "new_h2_CALM_XGBoost_N5_t2", "algo": "XGBoost", "regime": "CALM", "horizon": 2, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "Brent_Oil_FRED_ret_5d", "XOM_ret_1d", "DAX_Germany_zscore_60d"], "is_new": true}, {"model_id": "new_h2_CALM_XGBoost_N5_t3", "algo": "XGBoost", "regime": "CALM", "horizon": 2, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "PFE_ret_1d", "SJM_JM_Smucker_ret_5d", "CTAS_Cintas_vol_20d"], "is_new": true}, {"model_id": "new_h2_CALM_XGBoost_N5_t4", "algo": "XGBoost", "regime": "CALM", "horizon": 2, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "LMT_LockheedMartin_ret_1d", "spx_vol_5d", "SJM_JM_Smucker_ret_1d"], "is_new": true}, {"model_id": "new_h2_CALM_XGBoost_N5_t5", "algo": "XGBoost", "regime": "CALM", "horizon": 2, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "MSTR_Bitcoin3_ret_1d", "MO_AltriaMG_ret_1d", "M_Macys_vol_20d"], "is_new": true}, {"model_id": "new_h2_CALM_XGBoost_N5_t6", "algo": "XGBoost", "regime": "CALM", "horizon": 2, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "US5Y_Rate_ret_5d", "CPB_CampbellSoup_ret_20d", "DHR_ret_1d"], "is_new": true}, {"model_id": "new_h2_CALM_XGBoost_N5_t7", "algo": "XGBoost", "regime": "CALM", "horizon": 2, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "XLV_Health_zscore_60d", "SLB_Schlumberger_ret_1d", "VRP_ma5"], "is_new": true}, {"model_id": "new_h2_CALM_XGBoost_N8_t0", "algo": "XGBoost", "regime": "CALM", "horizon": 2, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWM_Malaysia_ret_1d", "XLB_Materials_zscore_60d", "GE_ret_1d", "vix_mean_abs_ret_5d", "ORCL_vol_20d", "XOM_ret_20d"], "is_new": true}, {"model_id": "new_h2_CALM_XGBoost_N8_t1", "algo": "XGBoost", "regime": "CALM", "horizon": 2, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "DE_Deere_vol_20d", "US6M_Rate_ret_20d", "XLF_Fin_vol_20d", "AMT_AmericanTower_ret_1d", "PAYX_Paychex_ret_20d", "GD_GeneralDynamics_zscore_60d"], "is_new": true}, {"model_id": "new_h2_CALM_XGBoost_N8_t2", "algo": "XGBoost", "regime": "CALM", "horizon": 2, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "NWL_Newell_ret_20d", "VRP_ma5", "HangSeng_HK_ret_5d", "BTI_BritishAmerican_ret_20d", "ORCL_zscore_60d", "US3Y_Rate_ret_5d"], "is_new": true}, {"model_id": "new_h2_CALM_XGBoost_N8_t3", "algo": "XGBoost", "regime": "CALM", "horizon": 2, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "US30Y_Rate_ret_20d", "3M_ret_5d", "US1Y_Rate_ret_20d", "VVIX_ret_20d", "PFE_ret_1d", "HangSeng_HK_vol_20d"], "is_new": true}, {"model_id": "new_h2_CALM_XGBoost_N8_t4", "algo": "XGBoost", "regime": "CALM", "horizon": 2, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "XLY_Disc_vol_20d", "US3Y_Rate_ret_5d", "HD_ret_20d", "Michigan_Sentiment_ret_20d", "GILD_Gilead_ret_20d", "Brent_Oil_FRED_ret_5d"], "is_new": true}, {"model_id": "new_h2_CALM_XGBoost_N8_t5", "algo": "XGBoost", "regime": "CALM", "horizon": 2, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "NFCI_ret_5d", "PG_ret_20d", "EWS_Singapore_ret_5d", "HD_ret_5d", "PFE_ret_1d", "SPY_zscore_60d"], "is_new": true}, {"model_id": "new_h2_CALM_XGBoost_N8_t6", "algo": "XGBoost", "regime": "CALM", "horizon": 2, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "US6M_Rate_ret_20d", "TM_Telephone_ret_1d", "TGT_Target_zscore_60d", "CPB_CampbellSoup_zscore_60d", "MSTR_Bitcoin3_ret_20d", "HD_zscore_60d"], "is_new": true}, {"model_id": "new_h2_CALM_XGBoost_N8_t7", "algo": "XGBoost", "regime": "CALM", "horizon": 2, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "INTC_ret_1d", "AMD_ret_1d", "XLB_Materials_zscore_60d", "ENB_EnbridgeInc_ret_1d", "spx_vol_5d", "heston_var_ev_h3"], "is_new": true}, {"model_id": "new_h2_CALM_XGBoost_N10_t0", "algo": "XGBoost", "regime": "CALM", "horizon": 2, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EOG_EOGResources_ret_5d", "Core_PCE_zscore_60d", "ASX_Australia_ret_5d", "EMR_Emerson_ret_20d", "M_Macys_vol_20d", "DHR_ret_1d", "PFE_ret_1d", "AVB_AvalonBay_zscore_60d"], "is_new": true}, {"model_id": "new_h2_CALM_XGBoost_N10_t1", "algo": "XGBoost", "regime": "CALM", "horizon": 2, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "AMGN_Amgen_ret_1d", "TXN_vol_20d", "EQIX_Equinix_ret_5d", "GE_ret_1d", "TGT_Target_zscore_60d", "EWY_Korea_zscore_60d", "LOW_Lowes_ret_20d", "T10Y2Y_Spread_ret_5d"], "is_new": true}, {"model_id": "new_h2_CALM_XGBoost_N10_t2", "algo": "XGBoost", "regime": "CALM", "horizon": 2, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "vix_acceleration_1d", "AVB_AvalonBay_zscore_60d", "IYM_BasicMaterials_ret_20d", "SBUX_ret_5d", "HangSeng_HK_vol_20d", "TM_Telephone_vol_20d", "US1Y_Rate_ret_20d", "EWQ_France_ret_20d"], "is_new": true}, {"model_id": "new_h2_CALM_XGBoost_N10_t3", "algo": "XGBoost", "regime": "CALM", "horizon": 2, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "US1Y_Rate_ret_20d", "LLY_zscore_60d", "NOC_Northrop_ret_20d", "EWQ_France_ret_20d", "AMD_ret_5d", "DOW_Price_zscore_60d", "PAYX_Paychex_vol_20d", "MO_AltriaMG_ret_1d"], "is_new": true}, {"model_id": "new_h2_CALM_XGBoost_N10_t4", "algo": "XGBoost", "regime": "CALM", "horizon": 2, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "AXP_Amex_ret_20d", "LLY_zscore_60d", "MS_MorganStanley_ret_1d", "Michigan_Sentiment_ret_20d", "CPB_CampbellSoup_vol_20d", "VRP_ma5", "VVIX_ret_20d", "LOW_Lowes_ret_20d"], "is_new": true}, {"model_id": "new_h2_CALM_XGBoost_N10_t5", "algo": "XGBoost", "regime": "CALM", "horizon": 2, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "MS_MorganStanley_ret_1d", "HD_zscore_60d", "CPB_CampbellSoup_ret_20d", "US1Y_Rate_ret_5d", "EXC_Exelon_zscore_60d", "US3M_Rate_vol_20d", "EWA_Australia_zscore_60d", "ORCL_zscore_60d"], "is_new": true}, {"model_id": "new_h2_CALM_XGBoost_N10_t6", "algo": "XGBoost", "regime": "CALM", "horizon": 2, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "US3M_Rate_zscore_60d", "BA_ret_1d", "SJM_JM_Smucker_ret_5d", "Retail_Sales_zscore_60d", "AMGN_Amgen_ret_1d", "spx_abs_ret_max_5d", "Core_PCE_zscore_60d", "TED_Spread_zscore_60d"], "is_new": true}, {"model_id": "new_h2_CALM_XGBoost_N10_t7", "algo": "XGBoost", "regime": "CALM", "horizon": 2, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "AORD_AUS_zscore_60d", "CTAS_Cintas_vol_20d", "gjr_condvar_h1", "EWG_Germany_vol_20d", "Nikkei_Japan_vol_20d", "MRK_Merck_zscore_60d", "DAX_Germany_vol_20d", "AXP_Amex_vol_20d"], "is_new": true}, {"model_id": "new_h2_CALM_XGBoost_N12_t0", "algo": "XGBoost", "regime": "CALM", "horizon": 2, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWY_Korea_zscore_60d", "DE_Deere_ret_5d", "WTI_Oil_FRED_zscore_60d", "VVIX_ret_20d", "AXP_Amex_ret_20d", "MS_MorganStanley_ret_5d", "SBUX_ret_5d", "GILD_Gilead_ret_20d", "vix_acceleration_1d", "JNJ_ret_1d"], "is_new": true}, {"model_id": "new_h2_CALM_XGBoost_N12_t1", "algo": "XGBoost", "regime": "CALM", "horizon": 2, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "HangSeng_HK_ret_5d", "ASX_Australia_vol_20d", "MSTR_Bitcoin3_ret_1d", "DHR_vol_20d", "DAX_Germany_zscore_60d", "Core_CPI_zscore_60d", "ITT_ITTInc_ret_5d", "LLY_zscore_60d", "XLK_Tech_zscore_60d", "VVIX_ret_20d"], "is_new": true}, {"model_id": "new_h2_CALM_XGBoost_N12_t2", "algo": "XGBoost", "regime": "CALM", "horizon": 2, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "NWL_Newell_ret_20d", "EQIX_Equinix_ret_5d", "ES_Evergy_ret_1d", "EQR_Equity_ret_1d", "DAX_Germany_vol_20d", "XLF_Fin_vol_20d", "AMD_ret_1d", "IYM_BasicMaterials_ret_20d", "EWQ_France_ret_20d", "EWL_Switzerland_vol_20d"], "is_new": true}, {"model_id": "new_h2_CALM_XGBoost_N12_t3", "algo": "XGBoost", "regime": "CALM", "horizon": 2, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EXC_Exelon_zscore_60d", "SJM_JM_Smucker_ret_1d", "HangSeng_HK_vol_20d", "Retail_Sales_zscore_60d", "MS_MorganStanley_ret_1d", "HangSeng_HK_ret_5d", "TM_Telephone_vol_20d", "HD_ret_5d", "IWM_SmallCap_vol_20d", "MRK_Merck_zscore_60d"], "is_new": true}, {"model_id": "new_h2_CALM_XGBoost_N12_t4", "algo": "XGBoost", "regime": "CALM", "horizon": 2, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EOG_EOGResources_ret_5d", "CPB_CampbellSoup_ret_20d", "EWM_Malaysia_zscore_60d", "TGT_Target_zscore_60d", "EWM_Malaysia_ret_1d", "INTC_ret_1d", "3M_vol_20d", "SBUX_zscore_60d", "CCI_CrownCastle_vol_20d", "DHR_ret_1d"], "is_new": true}, {"model_id": "new_h2_CALM_XGBoost_N12_t5", "algo": "XGBoost", "regime": "CALM", "horizon": 2, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "CPB_CampbellSoup_vol_20d", "ENB_EnbridgeInc_ret_1d", "VVIX_ret_20d", "BDX_Becton_Dickinson_ret_20d", "3M_ret_5d", "EWY_Korea_ret_20d", "MS_MorganStanley_ret_5d", "DAX_Germany_zscore_60d", "spx_momentum_3d", "ES_Evergy_ret_1d"], "is_new": true}, {"model_id": "new_h2_CALM_XGBoost_N12_t6", "algo": "XGBoost", "regime": "CALM", "horizon": 2, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "heston_var_ev_h3", "DHR_vol_20d", "heston_ev_h3", "PG_ret_20d", "SCHW_Schwab_ret_5d", "EFFR_vol_20d", "NWL_Newell_ret_20d", "GE_ret_1d", "TM_Telephone_ret_1d", "heston_var_ev_h5"], "is_new": true}, {"model_id": "new_h2_CALM_XGBoost_N12_t7", "algo": "XGBoost", "regime": "CALM", "horizon": 2, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "ITT_ITTInc_ret_5d", "US1Y_Rate_ret_20d", "IWM_SmallCap_vol_20d", "EOG_EOGResources_ret_5d", "EQR_Equity_ret_1d", "US7Y_Rate_ret_20d", "vix_mean_abs_ret_5d", "HangSeng_HK_vol_20d", "LLY_zscore_60d", "DHR_ret_1d"], "is_new": true}, {"model_id": "new_h2_CALM_XGBoost_N15_t0", "algo": "XGBoost", "regime": "CALM", "horizon": 2, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "Core_CPI_zscore_60d", "SJM_JM_Smucker_ret_5d", "IWM_SmallCap_vol_20d", "SBUX_zscore_60d", "MO_AltriaMG_ret_1d", "XLV_Health_zscore_60d", "CI_Cigna_vol_20d", "PAYX_Paychex_vol_20d", "BTI_BritishAmerican_ret_5d", "BDX_Becton_Dickinson_ret_20d", "Industrial_Production_zscore_60d", "BA_ret_1d", "TED_Spread_zscore_60d"], "is_new": true}, {"model_id": "new_h2_CALM_XGBoost_N15_t1", "algo": "XGBoost", "regime": "CALM", "horizon": 2, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "WTI_Oil_FRED_zscore_60d", "CMCSA_ret_1d", "EXC_Exelon_zscore_60d", "PAYX_Paychex_vol_20d", "PCAR_PaccarInc_ret_5d", "HangSeng_HK_vol_20d", "GD_GeneralDynamics_zscore_60d", "PPL_PPL_ret_1d", "EWM_Malaysia_vol_20d", "LUV_SouthwestAir_ret_5d", "heston_var_ev_h3", "EWJ_Japan_vol_20d", "LOW_Lowes_ret_20d"], "is_new": true}, {"model_id": "new_h2_CALM_XGBoost_N15_t2", "algo": "XGBoost", "regime": "CALM", "horizon": 2, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "CTAS_Cintas_vol_20d", "PG_ret_20d", "Core_PCE_zscore_60d", "LOW_Lowes_ret_20d", "US6M_Rate_ret_20d", "HangSeng_HK_ret_1d", "US3Y_Rate_ret_5d", "PFE_ret_1d", "EWG_Germany_ret_20d", "EXC_Exelon_zscore_60d", "HD_zscore_60d", "XLB_Materials_zscore_60d", "EWG_Germany_vol_20d"], "is_new": true}, {"model_id": "new_h2_CALM_XGBoost_N15_t3", "algo": "XGBoost", "regime": "CALM", "horizon": 2, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "SLB_Schlumberger_ret_5d", "TGT_Target_zscore_60d", "AORD_AUS_zscore_60d", "EWL_Switzerland_zscore_60d", "EWA_Australia_zscore_60d", "DAX_Germany_vol_20d", "TED_Spread_zscore_60d", "EWL_Switzerland_vol_20d", "NVDA_vol_20d", "CI_Cigna_vol_20d", "BTI_BritishAmerican_ret_5d", "EWG_Germany_ret_20d", "EWQ_France_zscore_60d"], "is_new": true}, {"model_id": "new_h2_CALM_XGBoost_N15_t4", "algo": "XGBoost", "regime": "CALM", "horizon": 2, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "SO_SouthernCo_ret_5d", "QQQ_vol_20d", "SLB_Schlumberger_ret_1d", "TGT_Target_zscore_60d", "EOG_EOGResources_vol_20d", "US1Y_Rate_ret_5d", "BTI_BritishAmerican_ret_5d", "LMT_LockheedMartin_vol_20d", "Retail_Sales_zscore_60d", "TXN_vol_20d", "EWC_Canada_zscore_60d", "AMD_ret_1d", "HD_ret_5d"], "is_new": true}, {"model_id": "new_h2_CALM_XGBoost_N15_t5", "algo": "XGBoost", "regime": "CALM", "horizon": 2, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "BTI_BritishAmerican_ret_20d", "INTC_ret_1d", "EQR_Equity_ret_1d", "DAX_Germany_vol_20d", "PAYX_Paychex_ret_20d", "HangSeng_HK_vol_20d", "NEE_NextEra_ret_20d", "IWM_SmallCap_vol_20d", "spx_vol_5d", "EWJ_Japan_vol_20d", "EWM_Malaysia_ret_1d", "SBUX_ret_5d", "SO_SouthernCo_ret_5d"], "is_new": true}, {"model_id": "new_h2_CALM_XGBoost_N15_t6", "algo": "XGBoost", "regime": "CALM", "horizon": 2, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "NWL_Newell_ret_20d", "Retail_Sales_zscore_60d", "EWA_Australia_ret_1d", "HD_zscore_60d", "HD_ret_1d", "MSTR_Bitcoin3_ret_20d", "SJM_JM_Smucker_ret_1d", "CLX_Clorox_vol_20d", "PAYX_Paychex_ret_20d", "MSTR_Bitcoin3_ret_1d", "EMR_Emerson_ret_20d", "VRP_ma5", "MS_MorganStanley_zscore_60d"], "is_new": true}, {"model_id": "new_h2_CALM_XGBoost_N15_t7", "algo": "XGBoost", "regime": "CALM", "horizon": 2, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWH_HongKong_ret_5d", "MS_MorganStanley_zscore_60d", "DE_Deere_vol_20d", "VOD_Vodafone_zscore_60d", "XLB_Materials_zscore_60d", "US5Y_Rate_ret_5d", "EXC_Exelon_ret_1d", "MSTR_Bitcoin3_ret_1d", "gjr_condvar_h1", "Nikkei_Japan_zscore_60d", "FedFunds_zscore_60d", "JNJ_ret_1d", "BA_ret_1d"], "is_new": true}, {"model_id": "new_h2_CALM_XGBoost_N20_t0", "algo": "XGBoost", "regime": "CALM", "horizon": 2, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "AXP_Amex_vol_20d", "TGT_Target_zscore_60d", "VVIX_ret_20d", "US1Y_Rate_ret_20d", "JNJ_ret_1d", "Retail_Sales_zscore_60d", "EWM_Malaysia_vol_20d", "US5Y_Rate_ret_5d", "AMGN_Amgen_ret_1d", "EWM_Malaysia_zscore_60d", "EMR_Emerson_ret_20d", "CPB_CampbellSoup_ret_20d", "CI_Cigna_vol_20d", "TM_Telephone_vol_20d", "NEE_NextEra_ret_20d", "spx_vol_5d", "T10Y2Y_Spread_ret_5d", "HD_zscore_60d"], "is_new": true}, {"model_id": "new_h2_CALM_XGBoost_N20_t1", "algo": "XGBoost", "regime": "CALM", "horizon": 2, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "GILD_Gilead_ret_20d", "SO_SouthernCo_ret_5d", "SPY_zscore_60d", "HangSeng_HK_ret_1d", "XLY_Disc_vol_20d", "vix_mean_abs_ret_5d", "XLV_Health_zscore_60d", "PFE_ret_1d", "US7Y_Rate_ret_20d", "MO_AltriaMG_ret_1d", "ORCL_zscore_60d", "EOG_EOGResources_ret_5d", "US30Y_Rate_ret_20d", "SBUX_zscore_60d", "IYM_BasicMaterials_ret_20d", "heston_ev_h3", "US3Y_Rate_ret_5d", "LMT_LockheedMartin_ret_1d"], "is_new": true}, {"model_id": "new_h2_CALM_XGBoost_N20_t2", "algo": "XGBoost", "regime": "CALM", "horizon": 2, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EQIX_Equinix_ret_5d", "DHR_vol_20d", "Brent_Oil_FRED_ret_20d", "ORCL_zscore_60d", "US7Y_Rate_ret_20d", "PG_ret_20d", "TM_Telephone_ret_1d", "XLF_Fin_vol_20d", "spx_vol_5d", "gjr_condvar_h1", "DHR_ret_1d", "HangSeng_HK_ret_1d", "TED_Spread_zscore_60d", "TM_Telephone_vol_20d", "EWA_Australia_ret_1d", "heston_var_ev_h3", "PAYX_Paychex_zscore_60d", "CCI_CrownCastle_vol_20d"], "is_new": true}, {"model_id": "new_h2_CALM_XGBoost_N20_t3", "algo": "XGBoost", "regime": "CALM", "horizon": 2, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "FedFunds_zscore_60d", "IYM_BasicMaterials_ret_20d", "CPB_CampbellSoup_ret_5d", "GD_GeneralDynamics_zscore_60d", "EWG_Germany_vol_20d", "NWL_Newell_ret_20d", "SJM_JM_Smucker_ret_1d", "EFFR_vol_20d", "T10Y2Y_Spread_ret_5d", "MS_MorganStanley_ret_5d", "CPB_CampbellSoup_vol_20d", "CI_Cigna_vol_20d", "EQR_Equity_ret_1d", "EWL_Switzerland_vol_20d", "heston_var_ev_h3", "AORD_AUS_zscore_60d", "EWH_HongKong_ret_5d", "TED_Spread_zscore_60d"], "is_new": true}, {"model_id": "new_h2_CALM_XGBoost_N20_t4", "algo": "XGBoost", "regime": "CALM", "horizon": 2, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "CCI_CrownCastle_vol_20d", "US5Y_Rate_ret_5d", "EWS_Singapore_ret_5d", "XLK_Tech_zscore_60d", "SBUX_zscore_60d", "TM_Telephone_ret_1d", "GILD_Gilead_ret_20d", "Industrial_Production_zscore_60d", "AMD_ret_1d", "AMT_AmericanTower_ret_1d", "XLY_Disc_vol_20d", "spx_momentum_3d", "US3Y_Rate_ret_5d", "gjr_condvar_h1", "ENB_EnbridgeInc_ret_1d", "CPB_CampbellSoup_vol_20d", "PAYX_Paychex_ret_20d", "PLD_Prologis_ret_5d"], "is_new": true}, {"model_id": "new_h2_CALM_XGBoost_N20_t5", "algo": "XGBoost", "regime": "CALM", "horizon": 2, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "DIS_vol_20d", "US30Y_Rate_ret_20d", "NWL_Newell_ret_20d", "US7Y_Rate_ret_20d", "EWH_HongKong_ret_5d", "EWL_Switzerland_zscore_60d", "MRK_Merck_zscore_60d", "CLX_Clorox_vol_20d", "XOM_ret_20d", "spx_momentum_3d", "INTC_ret_1d", "SLB_Schlumberger_ret_1d", "TM_Telephone_ret_1d", "AMT_AmericanTower_ret_1d", "PG_ret_20d", "AMD_ret_5d", "vix_mean_abs_ret_5d", "INTC_ret_5d"], "is_new": true}, {"model_id": "new_h2_CALM_XGBoost_N20_t6", "algo": "XGBoost", "regime": "CALM", "horizon": 2, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "DAX_Germany_vol_20d", "US5Y_Rate_ret_5d", "EWQ_France_zscore_60d", "US7Y_Rate_ret_20d", "CCI_CrownCastle_vol_20d", "AMD_ret_1d", "EWY_Korea_ret_20d", "TGT_Target_zscore_60d", "DAX_Germany_zscore_60d", "US3Y_Rate_ret_5d", "QQQ_vol_20d", "HangSeng_HK_vol_20d", "DE_Deere_vol_20d", "heston_var_ev_h7", "INTC_ret_1d", "LMT_LockheedMartin_ret_1d", "EWS_Singapore_ret_5d", "Nikkei_Japan_vol_20d"], "is_new": true}, {"model_id": "new_h2_CALM_XGBoost_N20_t7", "algo": "XGBoost", "regime": "CALM", "horizon": 2, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "SJM_JM_Smucker_ret_5d", "DAX_Germany_vol_20d", "AVB_AvalonBay_zscore_60d", "HD_ret_20d", "IWM_SmallCap_vol_20d", "BLK_BlackRock_zscore_60d", "DE_Deere_vol_20d", "HangSeng_HK_ret_1d", "EWJ_Japan_vol_20d", "PG_ret_20d", "EWG_Germany_vol_20d", "M_Macys_vol_20d", "BTI_BritishAmerican_ret_5d", "JNJ_ret_1d", "TED_Spread_zscore_60d", "SJM_JM_Smucker_ret_1d", "MSTR_Bitcoin3_ret_20d", "DHR_ret_1d"], "is_new": true}, {"model_id": "new_h2_CALM_XGBoost_N25_t0", "algo": "XGBoost", "regime": "CALM", "horizon": 2, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "SLB_Schlumberger_ret_5d", "WTI_Oil_FRED_zscore_60d", "M_Macys_vol_20d", "US3M_Rate_zscore_60d", "EWS_Singapore_ret_5d", "EWH_HongKong_ret_5d", "LMT_LockheedMartin_vol_20d", "EWQ_France_zscore_60d", "XLY_Disc_vol_20d", "BLK_BlackRock_zscore_60d", "3M_vol_20d", "XOM_ret_20d", "IYR_US_REIT2_zscore_60d", "TXN_vol_20d", "ITT_ITTInc_ret_5d", "Nikkei_Japan_zscore_60d", "CMCSA_ret_1d", "LUV_SouthwestAir_ret_5d", "HD_zscore_60d", "PAYX_Paychex_zscore_60d", "EWY_Korea_ret_20d", "CLX_Clorox_vol_20d", "XOM_ret_1d"], "is_new": true}, {"model_id": "new_h2_CALM_XGBoost_N25_t1", "algo": "XGBoost", "regime": "CALM", "horizon": 2, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "US3M_Rate_zscore_60d", "ES_Evergy_ret_1d", "LOW_Lowes_ret_20d", "LLY_zscore_60d", "Core_CPI_zscore_60d", "CPB_CampbellSoup_ret_5d", "MSTR_Bitcoin3_ret_20d", "HD_zscore_60d", "EWY_Korea_ret_20d", "MS_MorganStanley_ret_5d", "XLV_Health_zscore_60d", "AVB_AvalonBay_zscore_60d", "DE_Deere_vol_20d", "SLB_Schlumberger_ret_5d", "EWQ_France_zscore_60d", "PPL_PPL_ret_1d", "SBUX_vol_20d", "EQR_Equity_ret_1d", "PCAR_PaccarInc_ret_5d", "AORD_AUS_zscore_60d", "EFFR_vol_20d", "TXN_vol_20d", "NVDA_vol_20d"], "is_new": true}, {"model_id": "new_h2_CALM_XGBoost_N25_t2", "algo": "XGBoost", "regime": "CALM", "horizon": 2, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "JNJ_ret_1d", "EWY_Korea_ret_20d", "HUM_Humana_ret_5d", "EQIX_Equinix_ret_5d", "T10Y2Y_Spread_ret_5d", "CI_Cigna_vol_20d", "MSTR_Bitcoin3_ret_1d", "EXC_Exelon_ret_1d", "heston_var_ev_h5", "BTI_BritishAmerican_ret_5d", "NFCI_ret_5d", "Brent_Oil_FRED_ret_5d", "ASX_Australia_ret_5d", "CPB_CampbellSoup_vol_20d", "PG_ret_20d", "ITT_ITTInc_ret_5d", "EWG_Germany_vol_20d", "IYR_US_REIT2_zscore_60d", "TM_Telephone_ret_1d", "HangSeng_HK_ret_1d", "NEE_NextEra_ret_20d", "FedFunds_zscore_60d", "AVB_AvalonBay_zscore_60d"], "is_new": true}, {"model_id": "new_h2_CALM_XGBoost_N25_t3", "algo": "XGBoost", "regime": "CALM", "horizon": 2, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "vix_mean_abs_ret_5d", "US30Y_Rate_ret_20d", "TXN_vol_20d", "3M_ret_5d", "DE_Deere_vol_20d", "SBUX_zscore_60d", "EWJ_Japan_vol_20d", "AMD_ret_1d", "FedFunds_zscore_60d", "DHR_vol_20d", "SBUX_vol_20d", "NVDA_vol_20d", "EWH_HongKong_ret_5d", "NWL_Newell_ret_20d", "spx_abs_ret_max_5d", "SLB_Schlumberger_ret_5d", "CPB_CampbellSoup_vol_20d", "heston_var_ev_h3", "EXC_Exelon_zscore_60d", "GE_ret_1d", "PCAR_PaccarInc_ret_5d", "Core_CPI_zscore_60d", "TED_Spread_zscore_60d"], "is_new": true}, {"model_id": "new_h2_CALM_XGBoost_N25_t4", "algo": "XGBoost", "regime": "CALM", "horizon": 2, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "AMT_AmericanTower_ret_1d", "US5Y_Rate_ret_5d", "NFCI_ret_5d", "vix_acceleration_1d", "AXP_Amex_ret_20d", "NVDA_vol_20d", "SCHW_Schwab_ret_5d", "M_Macys_vol_20d", "EXC_Exelon_zscore_60d", "XLF_Fin_vol_20d", "XOM_ret_20d", "3M_ret_5d", "US3M_Rate_zscore_60d", "EOG_EOGResources_vol_20d", "XLY_Disc_vol_20d", "SLB_Schlumberger_ret_1d", "TM_Telephone_ret_1d", "HD_ret_1d", "SLB_Schlumberger_ret_5d", "PCAR_PaccarInc_ret_5d", "GE_ret_1d", "INTC_ret_5d", "LMT_LockheedMartin_ret_1d"], "is_new": true}, {"model_id": "new_h2_CALM_XGBoost_N25_t5", "algo": "XGBoost", "regime": "CALM", "horizon": 2, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "DE_Deere_vol_20d", "SBUX_zscore_60d", "DAX_Germany_vol_20d", "PCAR_PaccarInc_ret_5d", "GD_GeneralDynamics_zscore_60d", "EWC_Canada_zscore_60d", "AMT_AmericanTower_ret_1d", "heston_var_ev_h5", "NWL_Newell_ret_20d", "US1Y_Rate_ret_20d", "US30Y_Rate_ret_20d", "Core_PCE_zscore_60d", "CPB_CampbellSoup_zscore_60d", "MSTR_Bitcoin3_ret_5d", "GE_ret_1d", "ASX_Australia_ret_5d", "EWG_Germany_ret_20d", "HangSeng_HK_vol_20d", "EWH_HongKong_ret_5d", "VRP_ma5", "SJM_JM_Smucker_ret_5d", "T10Y2Y_Spread_ret_5d", "BTI_BritishAmerican_ret_20d"], "is_new": true}, {"model_id": "new_h2_CALM_XGBoost_N25_t6", "algo": "XGBoost", "regime": "CALM", "horizon": 2, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "PAYX_Paychex_ret_20d", "hmm_p_stress", "LOW_Lowes_ret_20d", "SO_SouthernCo_ret_5d", "EWL_Switzerland_zscore_60d", "EWM_Malaysia_zscore_60d", "EWJ_Japan_vol_20d", "Michigan_Sentiment_ret_20d", "HD_ret_20d", "QQQ_vol_20d", "PAYX_Paychex_zscore_60d", "AMD_ret_5d", "EWY_Korea_zscore_60d", "CPB_CampbellSoup_ret_20d", "vix_acceleration_1d", "AMD_ret_1d", "XLB_Materials_zscore_60d", "XOM_ret_20d", "CTAS_Cintas_vol_20d", "LMT_LockheedMartin_ret_1d", "EWA_Australia_zscore_60d", "spx_momentum_3d", "NVDA_vol_20d"], "is_new": true}, {"model_id": "new_h2_CALM_XGBoost_N25_t7", "algo": "XGBoost", "regime": "CALM", "horizon": 2, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "DE_Deere_vol_20d", "LMT_LockheedMartin_ret_1d", "FedFunds_zscore_60d", "ENB_EnbridgeInc_ret_1d", "AVB_AvalonBay_zscore_60d", "spx_vol_5d", "BTI_BritishAmerican_ret_5d", "EOG_EOGResources_ret_5d", "EWM_Malaysia_zscore_60d", "CI_Cigna_vol_20d", "XLK_Tech_zscore_60d", "ASX_Australia_ret_5d", "LOW_Lowes_ret_20d", "PAYX_Paychex_zscore_60d", "PG_ret_20d", "vix_mean_abs_ret_5d", "3M_vol_20d", "GD_GeneralDynamics_zscore_60d", "HD_ret_1d", "BTI_BritishAmerican_ret_20d", "AMT_AmericanTower_ret_1d", "BDX_Becton_Dickinson_ret_20d", "SCHW_Schwab_ret_5d"], "is_new": true}, {"model_id": "new_h2_CALM_XGBoost_N30_t0", "algo": "XGBoost", "regime": "CALM", "horizon": 2, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EFFR_vol_20d", "GE_ret_1d", "MRK_Merck_zscore_60d", "EWQ_France_zscore_60d", "US1Y_Rate_ret_20d", "heston_var_ev_h5", "Brent_Oil_FRED_ret_5d", "BLK_BlackRock_zscore_60d", "XOM_ret_1d", "SBUX_ret_5d", "XLV_Health_zscore_60d", "IBEX_Spain_ret_20d", "CPB_CampbellSoup_ret_5d", "DHR_vol_20d", "XLY_Disc_vol_20d", "GILD_Gilead_ret_20d", "TXN_vol_20d", "PLD_Prologis_ret_5d", "AORD_AUS_zscore_60d", "VOD_Vodafone_zscore_60d", "US1Y_Rate_ret_5d", "QQQ_vol_20d", "DOW_Price_zscore_60d", "MS_MorganStanley_ret_1d", "EXC_Exelon_ret_1d", "NEE_NextEra_ret_20d", "EWG_Germany_vol_20d", "EQR_Equity_ret_1d"], "is_new": true}, {"model_id": "new_h2_CALM_XGBoost_N30_t1", "algo": "XGBoost", "regime": "CALM", "horizon": 2, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "HangSeng_HK_ret_1d", "DHR_vol_20d", "ES_Evergy_ret_1d", "HD_zscore_60d", "AMD_ret_5d", "FedFunds_zscore_60d", "heston_var_ev_h7", "EWJ_Japan_vol_20d", "EOG_EOGResources_vol_20d", "NVDA_vol_20d", "CPB_CampbellSoup_ret_5d", "CPB_CampbellSoup_zscore_60d", "TGT_Target_zscore_60d", "BA_ret_1d", "ASX_Australia_vol_20d", "EWA_Australia_zscore_60d", "HangSeng_HK_vol_20d", "MSTR_Bitcoin3_ret_20d", "TED_Spread_vol_20d", "Nikkei_Japan_zscore_60d", "gjr_condvar_h1", "XLB_Materials_zscore_60d", "EWL_Switzerland_vol_20d", "AXP_Amex_vol_20d", "QQQ_vol_20d", "spx_abs_ret_max_5d", "DOW_Price_zscore_60d", "TXN_vol_20d"], "is_new": true}, {"model_id": "new_h2_CALM_XGBoost_N30_t2", "algo": "XGBoost", "regime": "CALM", "horizon": 2, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "DOW_Price_zscore_60d", "ASX_Australia_ret_5d", "vix_acceleration_1d", "VVIX_ret_20d", "IYR_US_REIT2_zscore_60d", "EWQ_France_ret_20d", "MS_MorganStanley_ret_5d", "NOC_Northrop_ret_20d", "DE_Deere_vol_20d", "BTI_BritishAmerican_ret_5d", "US3Y_Rate_ret_5d", "LOW_Lowes_ret_5d", "EWC_Canada_zscore_60d", "NEE_NextEra_ret_20d", "TXN_vol_20d", "US7Y_Rate_ret_20d", "AORD_AUS_zscore_60d", "AMD_ret_1d", "Industrial_Production_zscore_60d", "PLD_Prologis_ret_5d", "QQQ_vol_20d", "INTC_ret_1d", "XLF_Fin_vol_20d", "BLK_BlackRock_zscore_60d", "CPB_CampbellSoup_ret_20d", "SCHW_Schwab_ret_5d", "EWS_Singapore_ret_5d", "heston_var_ev_h3"], "is_new": true}, {"model_id": "new_h2_CALM_XGBoost_N30_t3", "algo": "XGBoost", "regime": "CALM", "horizon": 2, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWL_Switzerland_vol_20d", "XLV_Health_zscore_60d", "PG_ret_20d", "CLX_Clorox_vol_20d", "EWA_Australia_ret_1d", "ASX_Australia_ret_5d", "PAYX_Paychex_zscore_60d", "EOG_EOGResources_ret_5d", "EMR_Emerson_ret_20d", "CMCSA_ret_1d", "CPB_CampbellSoup_zscore_60d", "EWG_Germany_vol_20d", "TM_Telephone_vol_20d", "MSTR_Bitcoin3_ret_20d", "US3M_Rate_vol_20d", "AVB_AvalonBay_zscore_60d", "SLB_Schlumberger_ret_5d", "PPL_PPL_ret_1d", "Brent_Oil_FRED_ret_5d", "MS_MorganStanley_ret_1d", "SCHW_Schwab_ret_5d", "DHR_vol_20d", "ES_Evergy_ret_1d", "Core_PCE_zscore_60d", "AMZN_ret_5d", "EWY_Korea_zscore_60d", "INTC_ret_1d", "HD_ret_1d"], "is_new": true}, {"model_id": "new_h2_CALM_XGBoost_N30_t4", "algo": "XGBoost", "regime": "CALM", "horizon": 2, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "CTAS_Cintas_vol_20d", "US3M_Rate_zscore_60d", "ENB_EnbridgeInc_ret_1d", "T_ret_1d", "BTI_BritishAmerican_ret_5d", "QQQ_vol_20d", "3M_vol_20d", "PCAR_PaccarInc_ret_5d", "EXC_Exelon_ret_1d", "HangSeng_HK_vol_20d", "EWS_Singapore_ret_5d", "NEE_NextEra_ret_20d", "BA_ret_1d", "DIS_vol_20d", "EWM_Malaysia_ret_1d", "HangSeng_HK_ret_5d", "HUM_Humana_ret_5d", "DE_Deere_ret_5d", "ES_Evergy_ret_1d", "LOW_Lowes_ret_5d", "DHR_vol_20d", "CMCSA_ret_1d", "MS_MorganStanley_zscore_60d", "AMD_ret_5d", "VRP_ma5", "JNJ_ret_1d", "EQR_Equity_ret_1d", "EWJ_Japan_vol_20d"], "is_new": true}, {"model_id": "new_h2_CALM_XGBoost_N30_t5", "algo": "XGBoost", "regime": "CALM", "horizon": 2, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "vix_acceleration_1d", "AMGN_Amgen_ret_1d", "US7Y_Rate_ret_20d", "GE_ret_1d", "HangSeng_HK_vol_20d", "PAYX_Paychex_vol_20d", "QQQ_vol_20d", "EQR_Equity_ret_1d", "MSTR_Bitcoin3_ret_1d", "XLV_Health_zscore_60d", "AMT_AmericanTower_ret_1d", "EWH_HongKong_ret_5d", "heston_var_ev_h5", "NVDA_vol_20d", "LMT_LockheedMartin_vol_20d", "INTC_ret_1d", "CPB_CampbellSoup_ret_20d", "EWM_Malaysia_ret_1d", "DE_Deere_ret_5d", "PPL_PPL_ret_1d", "NEE_NextEra_ret_20d", "CCI_CrownCastle_vol_20d", "PLD_Prologis_ret_5d", "HangSeng_HK_ret_1d", "US1Y_Rate_ret_5d", "AMZN_ret_5d", "DE_Deere_vol_20d", "spx_momentum_3d"], "is_new": true}, {"model_id": "new_h2_CALM_XGBoost_N30_t6", "algo": "XGBoost", "regime": "CALM", "horizon": 2, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "MSTR_Bitcoin3_ret_20d", "ASX_Australia_vol_20d", "DE_Deere_ret_5d", "CPB_CampbellSoup_zscore_60d", "spx_momentum_3d", "US5Y_Rate_ret_5d", "CTAS_Cintas_vol_20d", "INTC_ret_5d", "TM_Telephone_ret_1d", "US6M_Rate_ret_20d", "CMCSA_ret_1d", "EOG_EOGResources_ret_5d", "Nikkei_Japan_zscore_60d", "TM_Telephone_vol_20d", "Michigan_Sentiment_ret_20d", "HD_ret_5d", "XLF_Fin_vol_20d", "heston_var_ev_h3", "NFCI_ret_5d", "EXC_Exelon_ret_1d", "Retail_Sales_zscore_60d", "US1Y_Rate_ret_5d", "Nikkei_Japan_vol_20d", "SO_SouthernCo_ret_5d", "NVDA_vol_20d", "DAX_Germany_zscore_60d", "EWJ_Japan_vol_20d", "PG_ret_20d"], "is_new": true}, {"model_id": "new_h2_CALM_XGBoost_N30_t7", "algo": "XGBoost", "regime": "CALM", "horizon": 2, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "IBEX_Spain_ret_20d", "PCAR_PaccarInc_ret_5d", "EXC_Exelon_zscore_60d", "AMZN_ret_5d", "SLB_Schlumberger_ret_5d", "EWM_Malaysia_zscore_60d", "ORCL_zscore_60d", "US1Y_Rate_ret_5d", "CLX_Clorox_vol_20d", "US3M_Rate_vol_20d", "CTAS_Cintas_vol_20d", "US3Y_Rate_ret_5d", "ASX_Australia_ret_5d", "SBUX_zscore_60d", "HD_zscore_60d", "US3M_Rate_zscore_60d", "DE_Deere_ret_5d", "CI_Cigna_vol_20d", "ORCL_vol_20d", "HangSeng_HK_ret_1d", "AXP_Amex_ret_20d", "AXP_Amex_vol_20d", "Michigan_Sentiment_ret_20d", "XOM_ret_20d", "TED_Spread_zscore_60d", "EWG_Germany_ret_20d", "MS_MorganStanley_ret_5d", "DAX_Germany_zscore_60d"], "is_new": true}, {"model_id": "new_h2_CALM_LightGBM_N5_t0", "algo": "LightGBM", "regime": "CALM", "horizon": 2, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "TM_Telephone_vol_20d", "GD_GeneralDynamics_zscore_60d", "SLB_Schlumberger_ret_5d"], "is_new": true}, {"model_id": "new_h2_CALM_LightGBM_N5_t1", "algo": "LightGBM", "regime": "CALM", "horizon": 2, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "SLB_Schlumberger_ret_1d", "EFFR_vol_20d", "Core_PCE_zscore_60d"], "is_new": true}, {"model_id": "new_h2_CALM_LightGBM_N5_t2", "algo": "LightGBM", "regime": "CALM", "horizon": 2, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "Nikkei_Japan_zscore_60d", "ES_Evergy_ret_1d", "DIS_vol_20d"], "is_new": true}, {"model_id": "new_h2_CALM_LightGBM_N5_t3", "algo": "LightGBM", "regime": "CALM", "horizon": 2, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "ENB_EnbridgeInc_ret_1d", "EWG_Germany_ret_20d", "SBUX_zscore_60d"], "is_new": true}, {"model_id": "new_h2_CALM_LightGBM_N5_t4", "algo": "LightGBM", "regime": "CALM", "horizon": 2, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "IYM_BasicMaterials_ret_20d", "3M_ret_5d", "EWH_HongKong_ret_5d"], "is_new": true}, {"model_id": "new_h2_CALM_LightGBM_N5_t5", "algo": "LightGBM", "regime": "CALM", "horizon": 2, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "AMGN_Amgen_ret_1d", "T_ret_1d", "SPY_zscore_60d"], "is_new": true}, {"model_id": "new_h2_CALM_LightGBM_N5_t6", "algo": "LightGBM", "regime": "CALM", "horizon": 2, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "PCAR_PaccarInc_ret_5d", "EWQ_France_zscore_60d", "VRP_ma5"], "is_new": true}, {"model_id": "new_h2_CALM_LightGBM_N5_t7", "algo": "LightGBM", "regime": "CALM", "horizon": 2, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "DAX_Germany_vol_20d", "EWY_Korea_zscore_60d", "Core_PCE_zscore_60d"], "is_new": true}, {"model_id": "new_h2_CALM_LightGBM_N8_t0", "algo": "LightGBM", "regime": "CALM", "horizon": 2, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "MSTR_Bitcoin3_ret_5d", "US3M_Rate_zscore_60d", "XOM_ret_20d", "DOW_Price_zscore_60d", "EWQ_France_ret_20d", "DHR_ret_1d"], "is_new": true}, {"model_id": "new_h2_CALM_LightGBM_N8_t1", "algo": "LightGBM", "regime": "CALM", "horizon": 2, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "US6M_Rate_ret_20d", "TGT_Target_zscore_60d", "US5Y_Rate_ret_5d", "PFE_ret_1d", "XLY_Disc_vol_20d", "AXP_Amex_ret_20d"], "is_new": true}, {"model_id": "new_h2_CALM_LightGBM_N8_t2", "algo": "LightGBM", "regime": "CALM", "horizon": 2, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "TXN_vol_20d", "CI_Cigna_vol_20d", "T_ret_1d", "NWL_Newell_ret_20d", "Retail_Sales_zscore_60d", "EXC_Exelon_zscore_60d"], "is_new": true}, {"model_id": "new_h2_CALM_LightGBM_N8_t3", "algo": "LightGBM", "regime": "CALM", "horizon": 2, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "AMT_AmericanTower_ret_1d", "VVIX_ret_20d", "VOD_Vodafone_zscore_60d", "SO_SouthernCo_ret_5d", "GILD_Gilead_ret_20d", "EWL_Switzerland_zscore_60d"], "is_new": true}, {"model_id": "new_h2_CALM_LightGBM_N8_t4", "algo": "LightGBM", "regime": "CALM", "horizon": 2, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWS_Singapore_ret_5d", "PAYX_Paychex_vol_20d", "CI_Cigna_vol_20d", "CPB_CampbellSoup_ret_20d", "T10Y2Y_Spread_ret_5d", "INTC_ret_5d"], "is_new": true}, {"model_id": "new_h2_CALM_LightGBM_N8_t5", "algo": "LightGBM", "regime": "CALM", "horizon": 2, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "SJM_JM_Smucker_ret_5d", "DHR_vol_20d", "US3Y_Rate_ret_5d", "IBEX_Spain_ret_20d", "DOW_Price_zscore_60d", "AMGN_Amgen_ret_1d"], "is_new": true}, {"model_id": "new_h2_CALM_LightGBM_N8_t6", "algo": "LightGBM", "regime": "CALM", "horizon": 2, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "LOW_Lowes_ret_5d", "EWG_Germany_vol_20d", "ITT_ITTInc_ret_5d", "EWQ_France_zscore_60d", "HD_ret_1d", "NFCI_ret_5d"], "is_new": true}, {"model_id": "new_h2_CALM_LightGBM_N8_t7", "algo": "LightGBM", "regime": "CALM", "horizon": 2, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "MSTR_Bitcoin3_ret_1d", "US5Y_Rate_ret_5d", "XLF_Fin_vol_20d", "VVIX_ret_20d", "EWY_Korea_zscore_60d", "HD_ret_20d"], "is_new": true}, {"model_id": "new_h2_CALM_LightGBM_N10_t0", "algo": "LightGBM", "regime": "CALM", "horizon": 2, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "PAYX_Paychex_ret_20d", "DIS_vol_20d", "BA_ret_1d", "CPB_CampbellSoup_ret_20d", "BDX_Becton_Dickinson_ret_20d", "T_ret_1d", "DHR_ret_1d", "GD_GeneralDynamics_zscore_60d"], "is_new": true}, {"model_id": "new_h2_CALM_LightGBM_N10_t1", "algo": "LightGBM", "regime": "CALM", "horizon": 2, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "SLB_Schlumberger_ret_1d", "DOW_Price_zscore_60d", "DHR_ret_1d", "spx_abs_ret_max_5d", "EWL_Switzerland_zscore_60d", "SO_SouthernCo_ret_5d", "PAYX_Paychex_vol_20d", "VVIX_ret_20d"], "is_new": true}, {"model_id": "new_h2_CALM_LightGBM_N10_t2", "algo": "LightGBM", "regime": "CALM", "horizon": 2, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "DHR_vol_20d", "ASX_Australia_ret_5d", "TED_Spread_zscore_60d", "LLY_zscore_60d", "SJM_JM_Smucker_ret_5d", "BTI_BritishAmerican_ret_5d", "SJM_JM_Smucker_ret_1d", "FedFunds_zscore_60d"], "is_new": true}, {"model_id": "new_h2_CALM_LightGBM_N10_t3", "algo": "LightGBM", "regime": "CALM", "horizon": 2, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWM_Malaysia_vol_20d", "LOW_Lowes_ret_20d", "heston_var_ev_h3", "EWA_Australia_zscore_60d", "NFCI_ret_5d", "IBEX_Spain_ret_20d", "EFFR_vol_20d", "EWL_Switzerland_zscore_60d"], "is_new": true}, {"model_id": "new_h2_CALM_LightGBM_N10_t4", "algo": "LightGBM", "regime": "CALM", "horizon": 2, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "HUM_Humana_ret_5d", "CLX_Clorox_vol_20d", "EFFR_vol_20d", "XLY_Disc_vol_20d", "CPB_CampbellSoup_ret_20d", "US30Y_Rate_ret_20d", "DHR_ret_1d", "US1Y_Rate_ret_20d"], "is_new": true}, {"model_id": "new_h2_CALM_LightGBM_N10_t5", "algo": "LightGBM", "regime": "CALM", "horizon": 2, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "NVDA_vol_20d", "BTI_BritishAmerican_ret_5d", "XOM_ret_20d", "EQIX_Equinix_ret_5d", "gjr_condvar_h1", "IWM_SmallCap_vol_20d", "LLY_zscore_60d", "US3M_Rate_vol_20d"], "is_new": true}, {"model_id": "new_h2_CALM_LightGBM_N10_t6", "algo": "LightGBM", "regime": "CALM", "horizon": 2, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "ASX_Australia_ret_5d", "BLK_BlackRock_zscore_60d", "PG_ret_20d", "XLB_Materials_zscore_60d", "EOG_EOGResources_vol_20d", "EWL_Switzerland_zscore_60d", "BDX_Becton_Dickinson_ret_20d", "FedFunds_zscore_60d"], "is_new": true}, {"model_id": "new_h2_CALM_LightGBM_N10_t7", "algo": "LightGBM", "regime": "CALM", "horizon": 2, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "CI_Cigna_vol_20d", "DAX_Germany_zscore_60d", "US30Y_Rate_ret_20d", "EWS_Singapore_ret_5d", "T10Y2Y_Spread_ret_5d", "DHR_vol_20d", "PFE_ret_1d", "CLX_Clorox_vol_20d"], "is_new": true}, {"model_id": "new_h2_CALM_LightGBM_N12_t0", "algo": "LightGBM", "regime": "CALM", "horizon": 2, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "DAX_Germany_zscore_60d", "US3M_Rate_zscore_60d", "XOM_ret_20d", "Nikkei_Japan_vol_20d", "BTI_BritishAmerican_ret_5d", "LMT_LockheedMartin_ret_1d", "CLX_Clorox_vol_20d", "AORD_AUS_zscore_60d", "spx_abs_ret_max_5d", "FedFunds_zscore_60d"], "is_new": true}, {"model_id": "new_h2_CALM_LightGBM_N12_t1", "algo": "LightGBM", "regime": "CALM", "horizon": 2, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "Nikkei_Japan_vol_20d", "ORCL_vol_20d", "IWM_SmallCap_vol_20d", "EWM_Malaysia_ret_1d", "EXC_Exelon_zscore_60d", "IBEX_Spain_ret_20d", "MSTR_Bitcoin3_ret_20d", "AXP_Amex_ret_20d", "HangSeng_HK_vol_20d", "spx_abs_ret_max_5d"], "is_new": true}, {"model_id": "new_h2_CALM_LightGBM_N12_t2", "algo": "LightGBM", "regime": "CALM", "horizon": 2, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWL_Switzerland_zscore_60d", "heston_ev_h3", "HangSeng_HK_ret_1d", "MSTR_Bitcoin3_ret_5d", "AMZN_ret_5d", "CPB_CampbellSoup_zscore_60d", "XOM_ret_1d", "CLX_Clorox_vol_20d", "IYR_US_REIT2_zscore_60d", "EWQ_France_ret_20d"], "is_new": true}, {"model_id": "new_h2_CALM_LightGBM_N12_t3", "algo": "LightGBM", "regime": "CALM", "horizon": 2, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "IBEX_Spain_ret_20d", "SBUX_zscore_60d", "DOW_Price_zscore_60d", "EXC_Exelon_zscore_60d", "EWQ_France_ret_20d", "LMT_LockheedMartin_ret_1d", "US1Y_Rate_ret_5d", "NVDA_vol_20d", "CI_Cigna_vol_20d", "NEE_NextEra_ret_20d"], "is_new": true}, {"model_id": "new_h2_CALM_LightGBM_N12_t4", "algo": "LightGBM", "regime": "CALM", "horizon": 2, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "CPB_CampbellSoup_ret_5d", "DAX_Germany_zscore_60d", "PFE_ret_1d", "vix_mean_abs_ret_5d", "EQR_Equity_ret_1d", "MS_MorganStanley_ret_5d", "PAYX_Paychex_zscore_60d", "3M_ret_5d", "SLB_Schlumberger_ret_5d", "NVDA_vol_20d"], "is_new": true}, {"model_id": "new_h2_CALM_LightGBM_N12_t5", "algo": "LightGBM", "regime": "CALM", "horizon": 2, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EFFR_ret_1d", "DE_Deere_ret_5d", "GILD_Gilead_ret_20d", "Brent_Oil_FRED_ret_5d", "XOM_ret_1d", "PAYX_Paychex_ret_20d", "NEE_NextEra_ret_20d", "TXN_vol_20d", "SLB_Schlumberger_ret_5d", "GE_ret_1d"], "is_new": true}, {"model_id": "new_h2_CALM_LightGBM_N12_t6", "algo": "LightGBM", "regime": "CALM", "horizon": 2, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "CCI_CrownCastle_vol_20d", "spx_vol_5d", "EFFR_vol_20d", "US6M_Rate_ret_20d", "BLK_BlackRock_zscore_60d", "heston_var_ev_h3", "EWH_HongKong_ret_5d", "LOW_Lowes_ret_5d", "AMD_ret_5d", "ASX_Australia_vol_20d"], "is_new": true}, {"model_id": "new_h2_CALM_LightGBM_N12_t7", "algo": "LightGBM", "regime": "CALM", "horizon": 2, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "HangSeng_HK_ret_5d", "DIS_vol_20d", "DAX_Germany_zscore_60d", "EWQ_France_ret_20d", "INTC_ret_1d", "GE_ret_1d", "EWH_HongKong_ret_5d", "HangSeng_HK_vol_20d", "BA_ret_1d", "NVDA_vol_20d"], "is_new": true}, {"model_id": "new_h2_CALM_LightGBM_N15_t0", "algo": "LightGBM", "regime": "CALM", "horizon": 2, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "DOW_Price_zscore_60d", "CLX_Clorox_vol_20d", "Industrial_Production_zscore_60d", "heston_var_ev_h3", "Nikkei_Japan_zscore_60d", "EWY_Korea_zscore_60d", "XLB_Materials_zscore_60d", "LOW_Lowes_ret_20d", "EMR_Emerson_ret_20d", "EXC_Exelon_ret_1d", "US5Y_Rate_ret_5d", "EOG_EOGResources_vol_20d", "AMGN_Amgen_ret_1d"], "is_new": true}, {"model_id": "new_h2_CALM_LightGBM_N15_t1", "algo": "LightGBM", "regime": "CALM", "horizon": 2, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "DAX_Germany_vol_20d", "CPB_CampbellSoup_ret_5d", "vix_mean_abs_ret_5d", "XLK_Tech_zscore_60d", "SCHW_Schwab_ret_5d", "EWC_Canada_zscore_60d", "TXN_vol_20d", "T_ret_1d", "CCI_CrownCastle_vol_20d", "TED_Spread_zscore_60d", "DIS_vol_20d", "PG_ret_20d", "EWY_Korea_ret_20d"], "is_new": true}, {"model_id": "new_h2_CALM_LightGBM_N15_t2", "algo": "LightGBM", "regime": "CALM", "horizon": 2, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWM_Malaysia_ret_1d", "Industrial_Production_zscore_60d", "VOD_Vodafone_zscore_60d", "HD_ret_5d", "SJM_JM_Smucker_ret_5d", "EQIX_Equinix_ret_5d", "BTI_BritishAmerican_ret_20d", "TED_Spread_zscore_60d", "SPY_zscore_60d", "GILD_Gilead_ret_20d", "NVDA_vol_20d", "TM_Telephone_ret_1d", "AMT_AmericanTower_ret_1d"], "is_new": true}, {"model_id": "new_h2_CALM_LightGBM_N15_t3", "algo": "LightGBM", "regime": "CALM", "horizon": 2, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "LLY_zscore_60d", "CPB_CampbellSoup_ret_5d", "AXP_Amex_ret_20d", "EQR_Equity_ret_1d", "CMCSA_ret_1d", "EMR_Emerson_ret_20d", "MO_AltriaMG_ret_1d", "IBEX_Spain_ret_20d", "EWC_Canada_zscore_60d", "spx_momentum_3d", "HD_ret_1d", "CI_Cigna_vol_20d", "ES_Evergy_ret_1d"], "is_new": true}, {"model_id": "new_h2_CALM_LightGBM_N15_t4", "algo": "LightGBM", "regime": "CALM", "horizon": 2, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "BA_ret_1d", "IBEX_Spain_ret_20d", "XLK_Tech_zscore_60d", "TED_Spread_zscore_60d", "NWL_Newell_ret_20d", "3M_vol_20d", "EWQ_France_zscore_60d", "SJM_JM_Smucker_ret_1d", "DHR_vol_20d", "CPB_CampbellSoup_ret_5d", "heston_var_ev_h5", "SLB_Schlumberger_ret_1d", "MSTR_Bitcoin3_ret_5d"], "is_new": true}, {"model_id": "new_h2_CALM_LightGBM_N15_t5", "algo": "LightGBM", "regime": "CALM", "horizon": 2, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "BLK_BlackRock_zscore_60d", "EWG_Germany_ret_20d", "CPB_CampbellSoup_vol_20d", "VVIX_ret_20d", "PAYX_Paychex_vol_20d", "T_ret_1d", "VRP_ma5", "XOM_ret_1d", "XLB_Materials_zscore_60d", "AXP_Amex_vol_20d", "CCI_CrownCastle_vol_20d", "CLX_Clorox_vol_20d", "GILD_Gilead_ret_20d"], "is_new": true}, {"model_id": "new_h2_CALM_LightGBM_N15_t6", "algo": "LightGBM", "regime": "CALM", "horizon": 2, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "QQQ_vol_20d", "Core_CPI_zscore_60d", "EWQ_France_zscore_60d", "CPB_CampbellSoup_zscore_60d", "SJM_JM_Smucker_ret_1d", "CPB_CampbellSoup_ret_5d", "VOD_Vodafone_zscore_60d", "PLD_Prologis_ret_5d", "hmm_p_stress", "US1Y_Rate_ret_5d", "NOC_Northrop_ret_20d", "AXP_Amex_ret_20d", "CPB_CampbellSoup_vol_20d"], "is_new": true}, {"model_id": "new_h2_CALM_LightGBM_N15_t7", "algo": "LightGBM", "regime": "CALM", "horizon": 2, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "XLV_Health_zscore_60d", "BA_ret_1d", "SLB_Schlumberger_ret_1d", "EXC_Exelon_zscore_60d", "CPB_CampbellSoup_ret_5d", "US6M_Rate_ret_20d", "heston_var_ev_h7", "HD_ret_20d", "heston_ev_h3", "PLD_Prologis_ret_5d", "TED_Spread_vol_20d", "IYM_BasicMaterials_ret_20d", "LUV_SouthwestAir_ret_5d"], "is_new": true}, {"model_id": "new_h2_CALM_LightGBM_N20_t0", "algo": "LightGBM", "regime": "CALM", "horizon": 2, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "ORCL_zscore_60d", "JNJ_ret_1d", "EWM_Malaysia_ret_1d", "Industrial_Production_zscore_60d", "TED_Spread_zscore_60d", "ENB_EnbridgeInc_ret_1d", "Core_PCE_zscore_60d", "AMT_AmericanTower_ret_1d", "PPL_PPL_ret_1d", "US1Y_Rate_ret_5d", "PAYX_Paychex_ret_20d", "T10Y2Y_Spread_ret_5d", "3M_ret_5d", "DHR_ret_1d", "PLD_Prologis_ret_5d", "WTI_Oil_FRED_zscore_60d", "BDX_Becton_Dickinson_ret_20d", "EWM_Malaysia_vol_20d"], "is_new": true}, {"model_id": "new_h2_CALM_LightGBM_N20_t1", "algo": "LightGBM", "regime": "CALM", "horizon": 2, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "Nikkei_Japan_vol_20d", "AORD_AUS_zscore_60d", "US1Y_Rate_ret_20d", "SO_SouthernCo_ret_5d", "CTAS_Cintas_vol_20d", "XLV_Health_zscore_60d", "PPL_PPL_ret_1d", "EFFR_ret_1d", "DE_Deere_ret_5d", "EWG_Germany_ret_20d", "VVIX_ret_20d", "VOD_Vodafone_zscore_60d", "heston_ev_h3", "MSTR_Bitcoin3_ret_5d", "EWY_Korea_zscore_60d", "SBUX_ret_5d", "SBUX_zscore_60d", "Core_PCE_zscore_60d"], "is_new": true}, {"model_id": "new_h2_CALM_LightGBM_N20_t2", "algo": "LightGBM", "regime": "CALM", "horizon": 2, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWM_Malaysia_vol_20d", "AVB_AvalonBay_zscore_60d", "PAYX_Paychex_vol_20d", "NVDA_vol_20d", "US30Y_Rate_ret_20d", "CPB_CampbellSoup_zscore_60d", "3M_vol_20d", "EQR_Equity_ret_1d", "HD_ret_5d", "T_ret_1d", "AMD_ret_5d", "LUV_SouthwestAir_ret_5d", "LMT_LockheedMartin_vol_20d", "PLD_Prologis_ret_5d", "SJM_JM_Smucker_ret_5d", "ASX_Australia_ret_5d", "EOG_EOGResources_ret_5d", "hmm_p_stress"], "is_new": true}, {"model_id": "new_h2_CALM_LightGBM_N20_t3", "algo": "LightGBM", "regime": "CALM", "horizon": 2, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWJ_Japan_vol_20d", "MS_MorganStanley_zscore_60d", "ES_Evergy_ret_1d", "EMR_Emerson_ret_20d", "DAX_Germany_vol_20d", "MS_MorganStanley_ret_5d", "US7Y_Rate_ret_20d", "ASX_Australia_vol_20d", "MSTR_Bitcoin3_ret_5d", "XLB_Materials_zscore_60d", "vix_mean_abs_ret_5d", "MSTR_Bitcoin3_ret_20d", "US1Y_Rate_ret_20d", "LMT_LockheedMartin_vol_20d", "PCAR_PaccarInc_ret_5d", "EXC_Exelon_zscore_60d", "vix_acceleration_1d", "PPL_PPL_ret_1d"], "is_new": true}, {"model_id": "new_h2_CALM_LightGBM_N20_t4", "algo": "LightGBM", "regime": "CALM", "horizon": 2, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "BTI_BritishAmerican_ret_20d", "ORCL_vol_20d", "XLY_Disc_vol_20d", "Core_PCE_zscore_60d", "EWS_Singapore_ret_5d", "US5Y_Rate_ret_5d", "LUV_SouthwestAir_ret_5d", "DHR_vol_20d", "CLX_Clorox_vol_20d", "ITT_ITTInc_ret_5d", "BDX_Becton_Dickinson_ret_20d", "Core_CPI_zscore_60d", "NEE_NextEra_ret_20d", "HangSeng_HK_ret_1d", "DIS_vol_20d", "MS_MorganStanley_ret_1d", "Brent_Oil_FRED_ret_20d", "3M_vol_20d"], "is_new": true}, {"model_id": "new_h2_CALM_LightGBM_N20_t5", "algo": "LightGBM", "regime": "CALM", "horizon": 2, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "LUV_SouthwestAir_ret_5d", "AXP_Amex_ret_20d", "hmm_p_stress", "T10Y2Y_Spread_ret_5d", "EXC_Exelon_zscore_60d", "EFFR_vol_20d", "HD_ret_20d", "EWJ_Japan_vol_20d", "CTAS_Cintas_vol_20d", "XLY_Disc_vol_20d", "TM_Telephone_ret_1d", "EWL_Switzerland_zscore_60d", "US5Y_Rate_ret_5d", "DHR_ret_1d", "HD_ret_5d", "TM_Telephone_vol_20d", "HD_zscore_60d", "SJM_JM_Smucker_ret_5d"], "is_new": true}, {"model_id": "new_h2_CALM_LightGBM_N20_t6", "algo": "LightGBM", "regime": "CALM", "horizon": 2, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "HangSeng_HK_ret_5d", "CPB_CampbellSoup_zscore_60d", "NWL_Newell_ret_20d", "WTI_Oil_FRED_zscore_60d", "Brent_Oil_FRED_ret_5d", "AMD_ret_5d", "INTC_ret_1d", "EWM_Malaysia_vol_20d", "AXP_Amex_ret_20d", "DIS_vol_20d", "CPB_CampbellSoup_ret_5d", "MO_AltriaMG_ret_1d", "Brent_Oil_FRED_ret_20d", "MSTR_Bitcoin3_ret_1d", "QQQ_vol_20d", "LUV_SouthwestAir_ret_5d", "T_ret_1d", "MS_MorganStanley_ret_1d"], "is_new": true}, {"model_id": "new_h2_CALM_LightGBM_N20_t7", "algo": "LightGBM", "regime": "CALM", "horizon": 2, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWM_Malaysia_zscore_60d", "vix_acceleration_1d", "XLV_Health_zscore_60d", "IYM_BasicMaterials_ret_20d", "IBEX_Spain_ret_20d", "T_ret_1d", "CMCSA_ret_1d", "3M_ret_5d", "IWM_SmallCap_vol_20d", "AMGN_Amgen_ret_1d", "Core_PCE_zscore_60d", "GILD_Gilead_ret_20d", "SJM_JM_Smucker_ret_1d", "NVDA_vol_20d", "EXC_Exelon_zscore_60d", "EFFR_ret_1d", "SBUX_zscore_60d", "SLB_Schlumberger_ret_1d"], "is_new": true}, {"model_id": "new_h2_CALM_LightGBM_N25_t0", "algo": "LightGBM", "regime": "CALM", "horizon": 2, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "PPL_PPL_ret_1d", "EWY_Korea_ret_20d", "US1Y_Rate_ret_20d", "Industrial_Production_zscore_60d", "ITT_ITTInc_ret_5d", "CTAS_Cintas_vol_20d", "DHR_ret_1d", "TXN_vol_20d", "Brent_Oil_FRED_ret_5d", "PAYX_Paychex_vol_20d", "SCHW_Schwab_ret_5d", "IYM_BasicMaterials_ret_20d", "CPB_CampbellSoup_ret_5d", "DE_Deere_vol_20d", "EWM_Malaysia_zscore_60d", "VRP_ma5", "MRK_Merck_zscore_60d", "US3M_Rate_vol_20d", "SJM_JM_Smucker_ret_1d", "SLB_Schlumberger_ret_1d", "XOM_ret_1d", "JNJ_ret_1d", "LOW_Lowes_ret_5d"], "is_new": true}, {"model_id": "new_h2_CALM_LightGBM_N25_t1", "algo": "LightGBM", "regime": "CALM", "horizon": 2, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "vix_mean_abs_ret_5d", "WTI_Oil_FRED_zscore_60d", "PG_ret_20d", "EMR_Emerson_ret_20d", "EWM_Malaysia_vol_20d", "MSTR_Bitcoin3_ret_5d", "AMGN_Amgen_ret_1d", "EFFR_ret_1d", "HUM_Humana_ret_5d", "HD_zscore_60d", "ORCL_vol_20d", "MO_AltriaMG_ret_1d", "HD_ret_1d", "NWL_Newell_ret_20d", "spx_vol_5d", "PLD_Prologis_ret_5d", "DHR_vol_20d", "ASX_Australia_vol_20d", "LOW_Lowes_ret_20d", "BTI_BritishAmerican_ret_20d", "PAYX_Paychex_ret_20d", "CPB_CampbellSoup_vol_20d", "AMZN_ret_5d"], "is_new": true}, {"model_id": "new_h2_CALM_LightGBM_N25_t2", "algo": "LightGBM", "regime": "CALM", "horizon": 2, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "DAX_Germany_vol_20d", "LOW_Lowes_ret_5d", "MSTR_Bitcoin3_ret_5d", "FedFunds_zscore_60d", "hmm_p_stress", "SBUX_ret_5d", "TGT_Target_zscore_60d", "heston_var_ev_h5", "EWM_Malaysia_vol_20d", "spx_abs_ret_max_5d", "BLK_BlackRock_zscore_60d", "EWL_Switzerland_vol_20d", "CPB_CampbellSoup_ret_5d", "SBUX_vol_20d", "INTC_ret_5d", "HangSeng_HK_vol_20d", "heston_var_ev_h3", "TED_Spread_zscore_60d", "AMGN_Amgen_ret_1d", "ES_Evergy_ret_1d", "PAYX_Paychex_ret_20d", "HUM_Humana_ret_5d", "Brent_Oil_FRED_ret_20d"], "is_new": true}, {"model_id": "new_h2_CALM_LightGBM_N25_t3", "algo": "LightGBM", "regime": "CALM", "horizon": 2, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "MSTR_Bitcoin3_ret_5d", "AXP_Amex_vol_20d", "EMR_Emerson_ret_20d", "US3M_Rate_vol_20d", "EOG_EOGResources_vol_20d", "IYR_US_REIT2_zscore_60d", "EXC_Exelon_zscore_60d", "MS_MorganStanley_ret_5d", "TED_Spread_zscore_60d", "Michigan_Sentiment_ret_20d", "DIS_vol_20d", "PAYX_Paychex_zscore_60d", "ES_Evergy_ret_1d", "US1Y_Rate_ret_5d", "SPY_zscore_60d", "HangSeng_HK_ret_1d", "AXP_Amex_ret_20d", "PAYX_Paychex_vol_20d", "NOC_Northrop_ret_20d", "Nikkei_Japan_vol_20d", "EWY_Korea_zscore_60d", "DE_Deere_vol_20d", "AMD_ret_1d"], "is_new": true}, {"model_id": "new_h2_CALM_LightGBM_N25_t4", "algo": "LightGBM", "regime": "CALM", "horizon": 2, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "Retail_Sales_zscore_60d", "BTI_BritishAmerican_ret_5d", "EWG_Germany_vol_20d", "Industrial_Production_zscore_60d", "EWH_HongKong_ret_5d", "T_ret_1d", "M_Macys_vol_20d", "QQQ_vol_20d", "NOC_Northrop_ret_20d", "DHR_ret_1d", "LOW_Lowes_ret_20d", "XLB_Materials_zscore_60d", "DAX_Germany_zscore_60d", "EWL_Switzerland_vol_20d", "TED_Spread_zscore_60d", "Nikkei_Japan_zscore_60d", "SLB_Schlumberger_ret_1d", "heston_var_ev_h5", "BA_ret_1d", "NEE_NextEra_ret_20d", "AORD_AUS_zscore_60d", "FedFunds_zscore_60d", "SPY_zscore_60d"], "is_new": true}, {"model_id": "new_h2_CALM_LightGBM_N25_t5", "algo": "LightGBM", "regime": "CALM", "horizon": 2, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "BDX_Becton_Dickinson_ret_20d", "IYR_US_REIT2_zscore_60d", "MO_AltriaMG_ret_1d", "EWQ_France_ret_20d", "EXC_Exelon_zscore_60d", "3M_vol_20d", "LLY_zscore_60d", "AMZN_ret_5d", "TED_Spread_vol_20d", "XLB_Materials_zscore_60d", "DIS_vol_20d", "QQQ_vol_20d", "EWG_Germany_ret_20d", "US5Y_Rate_ret_5d", "NVDA_vol_20d", "CI_Cigna_vol_20d", "IWM_SmallCap_vol_20d", "VVIX_ret_20d", "PAYX_Paychex_zscore_60d", "Nikkei_Japan_vol_20d", "EXC_Exelon_ret_1d", "AMD_ret_1d", "INTC_ret_1d"], "is_new": true}, {"model_id": "new_h2_CALM_LightGBM_N25_t6", "algo": "LightGBM", "regime": "CALM", "horizon": 2, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "BDX_Becton_Dickinson_ret_20d", "QQQ_vol_20d", "SJM_JM_Smucker_ret_5d", "VVIX_ret_20d", "PG_ret_20d", "SJM_JM_Smucker_ret_1d", "Industrial_Production_zscore_60d", "LLY_zscore_60d", "EXC_Exelon_ret_1d", "EFFR_vol_20d", "ORCL_vol_20d", "Brent_Oil_FRED_ret_5d", "VOD_Vodafone_zscore_60d", "AORD_AUS_zscore_60d", "EWQ_France_zscore_60d", "TED_Spread_vol_20d", "EXC_Exelon_zscore_60d", "DIS_vol_20d", "EQR_Equity_ret_1d", "Retail_Sales_zscore_60d", "EWH_HongKong_ret_5d", "Michigan_Sentiment_ret_20d", "DE_Deere_ret_5d"], "is_new": true}, {"model_id": "new_h2_CALM_LightGBM_N25_t7", "algo": "LightGBM", "regime": "CALM", "horizon": 2, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EQIX_Equinix_ret_5d", "EOG_EOGResources_vol_20d", "heston_var_ev_h5", "DIS_vol_20d", "SBUX_ret_5d", "CPB_CampbellSoup_vol_20d", "CPB_CampbellSoup_zscore_60d", "Brent_Oil_FRED_ret_5d", "PG_ret_20d", "MO_AltriaMG_ret_1d", "SBUX_zscore_60d", "Nikkei_Japan_vol_20d", "XLF_Fin_vol_20d", "Core_PCE_zscore_60d", "gjr_condvar_h1", "heston_var_ev_h7", "NWL_Newell_ret_20d", "ENB_EnbridgeInc_ret_1d", "BDX_Becton_Dickinson_ret_20d", "HUM_Humana_ret_5d", "EFFR_vol_20d", "TXN_vol_20d", "MRK_Merck_zscore_60d"], "is_new": true}, {"model_id": "new_h2_CALM_LightGBM_N30_t0", "algo": "LightGBM", "regime": "CALM", "horizon": 2, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "US5Y_Rate_ret_5d", "US1Y_Rate_ret_20d", "EWM_Malaysia_zscore_60d", "ASX_Australia_ret_5d", "MRK_Merck_zscore_60d", "EOG_EOGResources_vol_20d", "CPB_CampbellSoup_vol_20d", "NWL_Newell_ret_20d", "US6M_Rate_ret_20d", "AMD_ret_1d", "EWH_HongKong_ret_5d", "US7Y_Rate_ret_20d", "EQIX_Equinix_ret_5d", "ES_Evergy_ret_1d", "AORD_AUS_zscore_60d", "EQR_Equity_ret_1d", "NFCI_ret_5d", "US3M_Rate_zscore_60d", "T_ret_1d", "AMD_ret_5d", "CPB_CampbellSoup_ret_20d", "GD_GeneralDynamics_zscore_60d", "XLY_Disc_vol_20d", "MS_MorganStanley_zscore_60d", "VOD_Vodafone_zscore_60d", "SCHW_Schwab_ret_5d", "Brent_Oil_FRED_ret_5d", "BLK_BlackRock_zscore_60d"], "is_new": true}, {"model_id": "new_h2_CALM_LightGBM_N30_t1", "algo": "LightGBM", "regime": "CALM", "horizon": 2, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "PG_ret_20d", "AMGN_Amgen_ret_1d", "EWL_Switzerland_vol_20d", "CCI_CrownCastle_vol_20d", "M_Macys_vol_20d", "EWA_Australia_zscore_60d", "PAYX_Paychex_vol_20d", "AVB_AvalonBay_zscore_60d", "JNJ_ret_1d", "AMD_ret_1d", "ASX_Australia_ret_5d", "HD_zscore_60d", "US7Y_Rate_ret_20d", "EWC_Canada_zscore_60d", "EWH_HongKong_ret_5d", "ES_Evergy_ret_1d", "NFCI_ret_5d", "PFE_ret_1d", "EQR_Equity_ret_1d", "EWY_Korea_zscore_60d", "HD_ret_1d", "MSTR_Bitcoin3_ret_1d", "US5Y_Rate_ret_5d", "EWM_Malaysia_ret_1d", "CTAS_Cintas_vol_20d", "ORCL_zscore_60d", "EFFR_vol_20d", "US3M_Rate_vol_20d"], "is_new": true}, {"model_id": "new_h2_CALM_LightGBM_N30_t2", "algo": "LightGBM", "regime": "CALM", "horizon": 2, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "US1Y_Rate_ret_5d", "3M_vol_20d", "TM_Telephone_vol_20d", "EWL_Switzerland_zscore_60d", "US3M_Rate_vol_20d", "ASX_Australia_ret_5d", "MS_MorganStanley_zscore_60d", "IWM_SmallCap_vol_20d", "EXC_Exelon_zscore_60d", "BLK_BlackRock_zscore_60d", "EFFR_vol_20d", "LMT_LockheedMartin_ret_1d", "XLV_Health_zscore_60d", "XLK_Tech_zscore_60d", "T_ret_1d", "heston_var_ev_h5", "CPB_CampbellSoup_zscore_60d", "XLB_Materials_zscore_60d", "spx_momentum_3d", "EWM_Malaysia_ret_1d", "DAX_Germany_zscore_60d", "PG_ret_20d", "gjr_condvar_h1", "SCHW_Schwab_ret_5d", "EQR_Equity_ret_1d", "spx_vol_5d", "CLX_Clorox_vol_20d", "US7Y_Rate_ret_20d"], "is_new": true}, {"model_id": "new_h2_CALM_LightGBM_N30_t3", "algo": "LightGBM", "regime": "CALM", "horizon": 2, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "LMT_LockheedMartin_ret_1d", "hmm_p_stress", "AXP_Amex_vol_20d", "HangSeng_HK_ret_1d", "US3M_Rate_vol_20d", "Nikkei_Japan_vol_20d", "BLK_BlackRock_zscore_60d", "DAX_Germany_zscore_60d", "Nikkei_Japan_zscore_60d", "NVDA_vol_20d", "PAYX_Paychex_zscore_60d", "MS_MorganStanley_zscore_60d", "IYM_BasicMaterials_ret_20d", "EOG_EOGResources_ret_5d", "PCAR_PaccarInc_ret_5d", "ENB_EnbridgeInc_ret_1d", "US1Y_Rate_ret_20d", "HangSeng_HK_ret_5d", "LOW_Lowes_ret_20d", "EWG_Germany_vol_20d", "VOD_Vodafone_zscore_60d", "SBUX_zscore_60d", "FedFunds_zscore_60d", "MRK_Merck_zscore_60d", "AMT_AmericanTower_ret_1d", "gjr_condvar_h1", "3M_ret_5d", "SLB_Schlumberger_ret_1d"], "is_new": true}, {"model_id": "new_h2_CALM_LightGBM_N30_t4", "algo": "LightGBM", "regime": "CALM", "horizon": 2, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "NVDA_vol_20d", "SCHW_Schwab_ret_5d", "LUV_SouthwestAir_ret_5d", "NFCI_ret_5d", "AMT_AmericanTower_ret_1d", "EOG_EOGResources_ret_5d", "AXP_Amex_ret_20d", "TXN_vol_20d", "AMD_ret_1d", "CPB_CampbellSoup_vol_20d", "US3M_Rate_vol_20d", "hmm_p_stress", "VVIX_ret_20d", "heston_var_ev_h7", "ENB_EnbridgeInc_ret_1d", "ES_Evergy_ret_1d", "Brent_Oil_FRED_ret_5d", "MRK_Merck_zscore_60d", "HangSeng_HK_ret_1d", "EWQ_France_ret_20d", "ASX_Australia_ret_5d", "HD_ret_1d", "EXC_Exelon_zscore_60d", "TM_Telephone_vol_20d", "US3Y_Rate_ret_5d", "VRP_ma5", "EWM_Malaysia_ret_1d", "EOG_EOGResources_vol_20d"], "is_new": true}, {"model_id": "new_h2_CALM_LightGBM_N30_t5", "algo": "LightGBM", "regime": "CALM", "horizon": 2, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWG_Germany_vol_20d", "EOG_EOGResources_vol_20d", "3M_vol_20d", "ENB_EnbridgeInc_ret_1d", "EWS_Singapore_ret_5d", "LOW_Lowes_ret_20d", "BTI_BritishAmerican_ret_5d", "AVB_AvalonBay_zscore_60d", "EFFR_ret_1d", "MRK_Merck_zscore_60d", "IYM_BasicMaterials_ret_20d", "LLY_zscore_60d", "BLK_BlackRock_zscore_60d", "Core_CPI_zscore_60d", "EWM_Malaysia_ret_1d", "Industrial_Production_zscore_60d", "HD_ret_20d", "gjr_condvar_h1", "ASX_Australia_ret_5d", "XLV_Health_zscore_60d", "Michigan_Sentiment_ret_20d", "US6M_Rate_ret_20d", "SLB_Schlumberger_ret_1d", "US1Y_Rate_ret_20d", "US30Y_Rate_ret_20d", "XLF_Fin_vol_20d", "EWH_HongKong_ret_5d", "EWA_Australia_zscore_60d"], "is_new": true}, {"model_id": "new_h2_CALM_LightGBM_N30_t6", "algo": "LightGBM", "regime": "CALM", "horizon": 2, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWY_Korea_ret_20d", "MS_MorganStanley_ret_5d", "EMR_Emerson_ret_20d", "VRP_ma5", "Nikkei_Japan_vol_20d", "DE_Deere_ret_5d", "3M_vol_20d", "TM_Telephone_ret_1d", "AMZN_ret_5d", "LLY_zscore_60d", "US3M_Rate_vol_20d", "heston_var_ev_h5", "EWM_Malaysia_zscore_60d", "DAX_Germany_vol_20d", "DAX_Germany_zscore_60d", "GD_GeneralDynamics_zscore_60d", "GE_ret_1d", "CMCSA_ret_1d", "spx_vol_5d", "US6M_Rate_ret_20d", "BDX_Becton_Dickinson_ret_20d", "PAYX_Paychex_ret_20d", "IYR_US_REIT2_zscore_60d", "vix_acceleration_1d", "INTC_ret_1d", "DE_Deere_vol_20d", "spx_momentum_3d", "Core_PCE_zscore_60d"], "is_new": true}, {"model_id": "new_h2_CALM_LightGBM_N30_t7", "algo": "LightGBM", "regime": "CALM", "horizon": 2, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWC_Canada_zscore_60d", "Brent_Oil_FRED_ret_5d", "SBUX_ret_5d", "PFE_ret_1d", "IBEX_Spain_ret_20d", "VOD_Vodafone_zscore_60d", "MRK_Merck_zscore_60d", "HD_ret_1d", "EFFR_vol_20d", "BTI_BritishAmerican_ret_20d", "AMD_ret_5d", "MSTR_Bitcoin3_ret_5d", "EQR_Equity_ret_1d", "VVIX_ret_20d", "GD_GeneralDynamics_zscore_60d", "CMCSA_ret_1d", "US3M_Rate_vol_20d", "EWG_Germany_vol_20d", "PAYX_Paychex_vol_20d", "JNJ_ret_1d", "CPB_CampbellSoup_ret_5d", "hmm_p_stress", "DE_Deere_ret_5d", "XOM_ret_1d", "MS_MorganStanley_zscore_60d", "WTI_Oil_FRED_zscore_60d", "vix_acceleration_1d", "EWM_Malaysia_vol_20d"], "is_new": true}, {"model_id": "new_h2_CALM_GradientBoosting_N5_t0", "algo": "GradientBoosting", "regime": "CALM", "horizon": 2, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "M_Macys_vol_20d", "DAX_Germany_zscore_60d", "NWL_Newell_ret_20d"], "is_new": true}, {"model_id": "new_h2_CALM_GradientBoosting_N5_t1", "algo": "GradientBoosting", "regime": "CALM", "horizon": 2, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "vix_acceleration_1d", "SLB_Schlumberger_ret_5d", "gjr_condvar_h1"], "is_new": true}, {"model_id": "new_h2_CALM_GradientBoosting_N5_t2", "algo": "GradientBoosting", "regime": "CALM", "horizon": 2, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "DHR_vol_20d", "AMZN_ret_5d", "XLY_Disc_vol_20d"], "is_new": true}, {"model_id": "new_h2_CALM_GradientBoosting_N5_t3", "algo": "GradientBoosting", "regime": "CALM", "horizon": 2, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "AMZN_ret_5d", "3M_vol_20d", "US3Y_Rate_ret_5d"], "is_new": true}, {"model_id": "new_h2_CALM_GradientBoosting_N5_t4", "algo": "GradientBoosting", "regime": "CALM", "horizon": 2, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "DHR_ret_1d", "GE_ret_1d", "BLK_BlackRock_zscore_60d"], "is_new": true}, {"model_id": "new_h2_CALM_GradientBoosting_N5_t5", "algo": "GradientBoosting", "regime": "CALM", "horizon": 2, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "LMT_LockheedMartin_vol_20d", "EWM_Malaysia_vol_20d", "IYM_BasicMaterials_ret_20d"], "is_new": true}, {"model_id": "new_h2_CALM_GradientBoosting_N5_t6", "algo": "GradientBoosting", "regime": "CALM", "horizon": 2, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "GE_ret_1d", "EWH_HongKong_ret_5d", "DHR_ret_1d"], "is_new": true}, {"model_id": "new_h2_CALM_GradientBoosting_N5_t7", "algo": "GradientBoosting", "regime": "CALM", "horizon": 2, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "vix_acceleration_1d", "AXP_Amex_vol_20d", "SPY_zscore_60d"], "is_new": true}, {"model_id": "new_h2_CALM_GradientBoosting_N8_t0", "algo": "GradientBoosting", "regime": "CALM", "horizon": 2, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EXC_Exelon_zscore_60d", "PLD_Prologis_ret_5d", "XLB_Materials_zscore_60d", "DAX_Germany_zscore_60d", "MRK_Merck_zscore_60d", "EWY_Korea_zscore_60d"], "is_new": true}, {"model_id": "new_h2_CALM_GradientBoosting_N8_t1", "algo": "GradientBoosting", "regime": "CALM", "horizon": 2, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EXC_Exelon_ret_1d", "XOM_ret_20d", "ITT_ITTInc_ret_5d", "TM_Telephone_vol_20d", "CTAS_Cintas_vol_20d", "VRP_ma5"], "is_new": true}, {"model_id": "new_h2_CALM_GradientBoosting_N8_t2", "algo": "GradientBoosting", "regime": "CALM", "horizon": 2, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "SCHW_Schwab_ret_5d", "AVB_AvalonBay_zscore_60d", "Nikkei_Japan_zscore_60d", "heston_ev_h3", "MSTR_Bitcoin3_ret_20d", "US5Y_Rate_ret_5d"], "is_new": true}, {"model_id": "new_h2_CALM_GradientBoosting_N8_t3", "algo": "GradientBoosting", "regime": "CALM", "horizon": 2, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "VOD_Vodafone_zscore_60d", "US1Y_Rate_ret_5d", "Nikkei_Japan_zscore_60d", "LMT_LockheedMartin_vol_20d", "CPB_CampbellSoup_ret_20d", "LOW_Lowes_ret_20d"], "is_new": true}, {"model_id": "new_h2_CALM_GradientBoosting_N8_t4", "algo": "GradientBoosting", "regime": "CALM", "horizon": 2, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "PLD_Prologis_ret_5d", "TM_Telephone_vol_20d", "vix_acceleration_1d", "HD_zscore_60d", "Nikkei_Japan_zscore_60d", "GILD_Gilead_ret_20d"], "is_new": true}, {"model_id": "new_h2_CALM_GradientBoosting_N8_t5", "algo": "GradientBoosting", "regime": "CALM", "horizon": 2, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "TM_Telephone_vol_20d", "EOG_EOGResources_ret_5d", "ITT_ITTInc_ret_5d", "ORCL_vol_20d", "Retail_Sales_zscore_60d", "DE_Deere_vol_20d"], "is_new": true}, {"model_id": "new_h2_CALM_GradientBoosting_N8_t6", "algo": "GradientBoosting", "regime": "CALM", "horizon": 2, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "MS_MorganStanley_zscore_60d", "SBUX_ret_5d", "DAX_Germany_zscore_60d", "spx_momentum_3d", "US7Y_Rate_ret_20d", "EOG_EOGResources_ret_5d"], "is_new": true}, {"model_id": "new_h2_CALM_GradientBoosting_N8_t7", "algo": "GradientBoosting", "regime": "CALM", "horizon": 2, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "MRK_Merck_zscore_60d", "CCI_CrownCastle_vol_20d", "US7Y_Rate_ret_20d", "MS_MorganStanley_ret_5d", "EOG_EOGResources_ret_5d", "US1Y_Rate_ret_5d"], "is_new": true}, {"model_id": "new_h2_CALM_GradientBoosting_N10_t0", "algo": "GradientBoosting", "regime": "CALM", "horizon": 2, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "AMD_ret_1d", "CPB_CampbellSoup_zscore_60d", "TGT_Target_zscore_60d", "EQR_Equity_ret_1d", "LMT_LockheedMartin_vol_20d", "spx_vol_5d", "heston_ev_h3", "Michigan_Sentiment_ret_20d"], "is_new": true}, {"model_id": "new_h2_CALM_GradientBoosting_N10_t1", "algo": "GradientBoosting", "regime": "CALM", "horizon": 2, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "US1Y_Rate_ret_20d", "NFCI_ret_5d", "EWA_Australia_ret_1d", "VOD_Vodafone_zscore_60d", "EWY_Korea_zscore_60d", "EQR_Equity_ret_1d", "heston_var_ev_h7", "AMD_ret_5d"], "is_new": true}, {"model_id": "new_h2_CALM_GradientBoosting_N10_t2", "algo": "GradientBoosting", "regime": "CALM", "horizon": 2, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "GILD_Gilead_ret_20d", "CLX_Clorox_vol_20d", "HD_ret_5d", "US3M_Rate_zscore_60d", "US3Y_Rate_ret_5d", "EOG_EOGResources_vol_20d", "VVIX_ret_20d", "AMD_ret_1d"], "is_new": true}, {"model_id": "new_h2_CALM_GradientBoosting_N10_t3", "algo": "GradientBoosting", "regime": "CALM", "horizon": 2, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "PAYX_Paychex_ret_20d", "AVB_AvalonBay_zscore_60d", "DE_Deere_vol_20d", "NEE_NextEra_ret_20d", "M_Macys_vol_20d", "3M_ret_5d", "FedFunds_zscore_60d", "JNJ_ret_1d"], "is_new": true}, {"model_id": "new_h2_CALM_GradientBoosting_N10_t4", "algo": "GradientBoosting", "regime": "CALM", "horizon": 2, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "ES_Evergy_ret_1d", "BTI_BritishAmerican_ret_5d", "Retail_Sales_zscore_60d", "HD_ret_20d", "heston_var_ev_h7", "AVB_AvalonBay_zscore_60d", "AMGN_Amgen_ret_1d", "EWM_Malaysia_zscore_60d"], "is_new": true}, {"model_id": "new_h2_CALM_GradientBoosting_N10_t5", "algo": "GradientBoosting", "regime": "CALM", "horizon": 2, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "US5Y_Rate_ret_5d", "EWL_Switzerland_zscore_60d", "EWS_Singapore_ret_5d", "MO_AltriaMG_ret_1d", "EWG_Germany_ret_20d", "TM_Telephone_ret_1d", "XLY_Disc_vol_20d", "EQIX_Equinix_ret_5d"], "is_new": true}, {"model_id": "new_h2_CALM_GradientBoosting_N10_t6", "algo": "GradientBoosting", "regime": "CALM", "horizon": 2, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "CMCSA_ret_1d", "CLX_Clorox_vol_20d", "VRP_ma5", "EWL_Switzerland_zscore_60d", "MSTR_Bitcoin3_ret_20d", "3M_vol_20d", "AXP_Amex_ret_20d", "NVDA_vol_20d"], "is_new": true}, {"model_id": "new_h2_CALM_GradientBoosting_N10_t7", "algo": "GradientBoosting", "regime": "CALM", "horizon": 2, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "spx_momentum_3d", "ASX_Australia_vol_20d", "ITT_ITTInc_ret_5d", "M_Macys_vol_20d", "HD_ret_1d", "CTAS_Cintas_vol_20d", "NVDA_vol_20d", "BDX_Becton_Dickinson_ret_20d"], "is_new": true}, {"model_id": "new_h2_CALM_GradientBoosting_N12_t0", "algo": "GradientBoosting", "regime": "CALM", "horizon": 2, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "HD_ret_20d", "EWL_Switzerland_vol_20d", "TED_Spread_vol_20d", "LMT_LockheedMartin_vol_20d", "AMD_ret_1d", "AMD_ret_5d", "US3Y_Rate_ret_5d", "IWM_SmallCap_vol_20d", "EWM_Malaysia_vol_20d", "PG_ret_20d"], "is_new": true}, {"model_id": "new_h2_CALM_GradientBoosting_N12_t1", "algo": "GradientBoosting", "regime": "CALM", "horizon": 2, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "CMCSA_ret_1d", "PAYX_Paychex_ret_20d", "NWL_Newell_ret_20d", "spx_momentum_3d", "DIS_vol_20d", "EWM_Malaysia_ret_1d", "CTAS_Cintas_vol_20d", "EWG_Germany_vol_20d", "EWC_Canada_zscore_60d", "NFCI_ret_5d"], "is_new": true}, {"model_id": "new_h2_CALM_GradientBoosting_N12_t2", "algo": "GradientBoosting", "regime": "CALM", "horizon": 2, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "CMCSA_ret_1d", "ORCL_vol_20d", "HangSeng_HK_ret_1d", "DHR_ret_1d", "EWG_Germany_ret_20d", "CPB_CampbellSoup_ret_5d", "MS_MorganStanley_ret_1d", "LMT_LockheedMartin_ret_1d", "GILD_Gilead_ret_20d", "HD_ret_5d"], "is_new": true}, {"model_id": "new_h2_CALM_GradientBoosting_N12_t3", "algo": "GradientBoosting", "regime": "CALM", "horizon": 2, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "PG_ret_20d", "PPL_PPL_ret_1d", "spx_momentum_3d", "MS_MorganStanley_ret_1d", "T10Y2Y_Spread_ret_5d", "LOW_Lowes_ret_5d", "Retail_Sales_zscore_60d", "NVDA_vol_20d", "ASX_Australia_ret_5d", "NWL_Newell_ret_20d"], "is_new": true}, {"model_id": "new_h2_CALM_GradientBoosting_N12_t4", "algo": "GradientBoosting", "regime": "CALM", "horizon": 2, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EQR_Equity_ret_1d", "XOM_ret_1d", "AXP_Amex_ret_20d", "HD_ret_1d", "Nikkei_Japan_vol_20d", "BTI_BritishAmerican_ret_5d", "QQQ_vol_20d", "IWM_SmallCap_vol_20d", "JNJ_ret_1d", "CPB_CampbellSoup_ret_5d"], "is_new": true}, {"model_id": "new_h2_CALM_GradientBoosting_N12_t5", "algo": "GradientBoosting", "regime": "CALM", "horizon": 2, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "SCHW_Schwab_ret_5d", "XLY_Disc_vol_20d", "XLV_Health_zscore_60d", "US3M_Rate_vol_20d", "EWA_Australia_ret_1d", "EOG_EOGResources_vol_20d", "HangSeng_HK_ret_1d", "EQR_Equity_ret_1d", "AVB_AvalonBay_zscore_60d", "EOG_EOGResources_ret_5d"], "is_new": true}, {"model_id": "new_h2_CALM_GradientBoosting_N12_t6", "algo": "GradientBoosting", "regime": "CALM", "horizon": 2, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "GD_GeneralDynamics_zscore_60d", "Brent_Oil_FRED_ret_5d", "SJM_JM_Smucker_ret_1d", "EOG_EOGResources_ret_5d", "EWL_Switzerland_zscore_60d", "LLY_zscore_60d", "Nikkei_Japan_vol_20d", "IYR_US_REIT2_zscore_60d", "EWC_Canada_zscore_60d", "SJM_JM_Smucker_ret_5d"], "is_new": true}, {"model_id": "new_h2_CALM_GradientBoosting_N12_t7", "algo": "GradientBoosting", "regime": "CALM", "horizon": 2, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "ENB_EnbridgeInc_ret_1d", "LOW_Lowes_ret_20d", "HUM_Humana_ret_5d", "TM_Telephone_ret_1d", "MO_AltriaMG_ret_1d", "EWM_Malaysia_ret_1d", "MSTR_Bitcoin3_ret_1d", "DHR_ret_1d", "NVDA_vol_20d", "US30Y_Rate_ret_20d"], "is_new": true}, {"model_id": "new_h2_CALM_GradientBoosting_N15_t0", "algo": "GradientBoosting", "regime": "CALM", "horizon": 2, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EOG_EOGResources_vol_20d", "EWC_Canada_zscore_60d", "SPY_zscore_60d", "ORCL_vol_20d", "GD_GeneralDynamics_zscore_60d", "BTI_BritishAmerican_ret_20d", "spx_abs_ret_max_5d", "AMD_ret_5d", "3M_ret_5d", "VVIX_ret_20d", "Industrial_Production_zscore_60d", "EWM_Malaysia_vol_20d", "SJM_JM_Smucker_ret_5d"], "is_new": true}, {"model_id": "new_h2_CALM_GradientBoosting_N15_t1", "algo": "GradientBoosting", "regime": "CALM", "horizon": 2, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EOG_EOGResources_vol_20d", "CMCSA_ret_1d", "MS_MorganStanley_ret_5d", "HD_ret_1d", "EWQ_France_ret_20d", "ITT_ITTInc_ret_5d", "VRP_ma5", "XLB_Materials_zscore_60d", "AMGN_Amgen_ret_1d", "ASX_Australia_ret_5d", "US6M_Rate_ret_20d", "EWL_Switzerland_zscore_60d", "Retail_Sales_zscore_60d"], "is_new": true}, {"model_id": "new_h2_CALM_GradientBoosting_N15_t2", "algo": "GradientBoosting", "regime": "CALM", "horizon": 2, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "AXP_Amex_vol_20d", "HangSeng_HK_vol_20d", "SBUX_zscore_60d", "heston_var_ev_h7", "heston_ev_h3", "NFCI_ret_5d", "CPB_CampbellSoup_ret_20d", "HUM_Humana_ret_5d", "EWL_Switzerland_zscore_60d", "PCAR_PaccarInc_ret_5d", "SLB_Schlumberger_ret_1d", "T10Y2Y_Spread_ret_5d", "FedFunds_zscore_60d"], "is_new": true}, {"model_id": "new_h2_CALM_GradientBoosting_N15_t3", "algo": "GradientBoosting", "regime": "CALM", "horizon": 2, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "MS_MorganStanley_zscore_60d", "MSTR_Bitcoin3_ret_1d", "CI_Cigna_vol_20d", "TM_Telephone_vol_20d", "PAYX_Paychex_zscore_60d", "hmm_p_stress", "LMT_LockheedMartin_ret_1d", "NOC_Northrop_ret_20d", "EXC_Exelon_ret_1d", "PLD_Prologis_ret_5d", "DE_Deere_ret_5d", "MS_MorganStanley_ret_1d", "EWM_Malaysia_zscore_60d"], "is_new": true}, {"model_id": "new_h2_CALM_GradientBoosting_N15_t4", "algo": "GradientBoosting", "regime": "CALM", "horizon": 2, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "CLX_Clorox_vol_20d", "SPY_zscore_60d", "MRK_Merck_zscore_60d", "XLF_Fin_vol_20d", "US3Y_Rate_ret_5d", "ASX_Australia_ret_5d", "BLK_BlackRock_zscore_60d", "DOW_Price_zscore_60d", "PAYX_Paychex_zscore_60d", "QQQ_vol_20d", "ENB_EnbridgeInc_ret_1d", "US5Y_Rate_ret_5d", "CTAS_Cintas_vol_20d"], "is_new": true}, {"model_id": "new_h2_CALM_GradientBoosting_N15_t5", "algo": "GradientBoosting", "regime": "CALM", "horizon": 2, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "PCAR_PaccarInc_ret_5d", "hmm_p_stress", "US30Y_Rate_ret_20d", "MSTR_Bitcoin3_ret_5d", "AMT_AmericanTower_ret_1d", "HD_zscore_60d", "EWQ_France_ret_20d", "LUV_SouthwestAir_ret_5d", "NFCI_ret_5d", "Nikkei_Japan_vol_20d", "ASX_Australia_ret_5d", "Industrial_Production_zscore_60d", "ASX_Australia_vol_20d"], "is_new": true}, {"model_id": "new_h2_CALM_GradientBoosting_N15_t6", "algo": "GradientBoosting", "regime": "CALM", "horizon": 2, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "NWL_Newell_ret_20d", "VRP_ma5", "SLB_Schlumberger_ret_5d", "EWS_Singapore_ret_5d", "EWJ_Japan_vol_20d", "AMZN_ret_5d", "MS_MorganStanley_zscore_60d", "SBUX_ret_5d", "heston_var_ev_h7", "ORCL_zscore_60d", "vix_acceleration_1d", "AMT_AmericanTower_ret_1d", "Michigan_Sentiment_ret_20d"], "is_new": true}, {"model_id": "new_h2_CALM_GradientBoosting_N15_t7", "algo": "GradientBoosting", "regime": "CALM", "horizon": 2, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EQR_Equity_ret_1d", "NVDA_vol_20d", "NFCI_ret_5d", "QQQ_vol_20d", "US5Y_Rate_ret_5d", "heston_var_ev_h5", "HD_ret_1d", "CCI_CrownCastle_vol_20d", "XLV_Health_zscore_60d", "AMGN_Amgen_ret_1d", "SO_SouthernCo_ret_5d", "PAYX_Paychex_zscore_60d", "SLB_Schlumberger_ret_5d"], "is_new": true}, {"model_id": "new_h2_CALM_GradientBoosting_N20_t0", "algo": "GradientBoosting", "regime": "CALM", "horizon": 2, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "SJM_JM_Smucker_ret_5d", "IYM_BasicMaterials_ret_20d", "WTI_Oil_FRED_zscore_60d", "AXP_Amex_ret_20d", "MS_MorganStanley_ret_1d", "PPL_PPL_ret_1d", "EWG_Germany_vol_20d", "US3Y_Rate_ret_5d", "NOC_Northrop_ret_20d", "HangSeng_HK_ret_5d", "EOG_EOGResources_ret_5d", "SO_SouthernCo_ret_5d", "XLY_Disc_vol_20d", "SBUX_ret_5d", "M_Macys_vol_20d", "AMZN_ret_5d", "INTC_ret_1d", "heston_var_ev_h3"], "is_new": true}, {"model_id": "new_h2_CALM_GradientBoosting_N20_t1", "algo": "GradientBoosting", "regime": "CALM", "horizon": 2, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "DHR_ret_1d", "Core_PCE_zscore_60d", "PCAR_PaccarInc_ret_5d", "BLK_BlackRock_zscore_60d", "FedFunds_zscore_60d", "heston_var_ev_h7", "US30Y_Rate_ret_20d", "NWL_Newell_ret_20d", "Brent_Oil_FRED_ret_5d", "TXN_vol_20d", "M_Macys_vol_20d", "PAYX_Paychex_ret_20d", "AORD_AUS_zscore_60d", "NEE_NextEra_ret_20d", "MS_MorganStanley_ret_5d", "EWG_Germany_vol_20d", "ES_Evergy_ret_1d", "BTI_BritishAmerican_ret_5d"], "is_new": true}, {"model_id": "new_h2_CALM_GradientBoosting_N20_t2", "algo": "GradientBoosting", "regime": "CALM", "horizon": 2, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWA_Australia_zscore_60d", "US6M_Rate_ret_20d", "EQIX_Equinix_ret_5d", "SLB_Schlumberger_ret_5d", "XLV_Health_zscore_60d", "CCI_CrownCastle_vol_20d", "US3M_Rate_zscore_60d", "TM_Telephone_ret_1d", "EFFR_vol_20d", "SBUX_zscore_60d", "Core_PCE_zscore_60d", "US30Y_Rate_ret_20d", "EFFR_ret_1d", "NVDA_vol_20d", "ITT_ITTInc_ret_5d", "EWG_Germany_vol_20d", "EWM_Malaysia_zscore_60d", "PG_ret_20d"], "is_new": true}, {"model_id": "new_h2_CALM_GradientBoosting_N20_t3", "algo": "GradientBoosting", "regime": "CALM", "horizon": 2, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "HangSeng_HK_vol_20d", "XLK_Tech_zscore_60d", "FedFunds_zscore_60d", "EOG_EOGResources_ret_5d", "Michigan_Sentiment_ret_20d", "VOD_Vodafone_zscore_60d", "HD_zscore_60d", "EFFR_vol_20d", "AORD_AUS_zscore_60d", "EWG_Germany_vol_20d", "VRP_ma5", "TM_Telephone_ret_1d", "vix_acceleration_1d", "EWQ_France_ret_20d", "CLX_Clorox_vol_20d", "IWM_SmallCap_vol_20d", "HangSeng_HK_ret_5d", "heston_var_ev_h3"], "is_new": true}, {"model_id": "new_h2_CALM_GradientBoosting_N20_t4", "algo": "GradientBoosting", "regime": "CALM", "horizon": 2, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "LOW_Lowes_ret_20d", "US3M_Rate_vol_20d", "XLK_Tech_zscore_60d", "NVDA_vol_20d", "PAYX_Paychex_zscore_60d", "AMZN_ret_5d", "MSTR_Bitcoin3_ret_20d", "WTI_Oil_FRED_zscore_60d", "EQIX_Equinix_ret_5d", "BTI_BritishAmerican_ret_20d", "XLV_Health_zscore_60d", "JNJ_ret_1d", "TM_Telephone_ret_1d", "ORCL_zscore_60d", "EWA_Australia_ret_1d", "3M_ret_5d", "MO_AltriaMG_ret_1d", "EXC_Exelon_zscore_60d"], "is_new": true}, {"model_id": "new_h2_CALM_GradientBoosting_N20_t5", "algo": "GradientBoosting", "regime": "CALM", "horizon": 2, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "US30Y_Rate_ret_20d", "SO_SouthernCo_ret_5d", "HD_ret_1d", "spx_momentum_3d", "EWS_Singapore_ret_5d", "BA_ret_1d", "DIS_vol_20d", "Brent_Oil_FRED_ret_5d", "PLD_Prologis_ret_5d", "TM_Telephone_ret_1d", "CTAS_Cintas_vol_20d", "NOC_Northrop_ret_20d", "PPL_PPL_ret_1d", "SJM_JM_Smucker_ret_5d", "3M_vol_20d", "MSTR_Bitcoin3_ret_5d", "heston_ev_h3", "MSTR_Bitcoin3_ret_1d"], "is_new": true}, {"model_id": "new_h2_CALM_GradientBoosting_N20_t6", "algo": "GradientBoosting", "regime": "CALM", "horizon": 2, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "JNJ_ret_1d", "heston_var_ev_h7", "TED_Spread_zscore_60d", "US6M_Rate_ret_20d", "BA_ret_1d", "CPB_CampbellSoup_vol_20d", "MSTR_Bitcoin3_ret_20d", "hmm_p_stress", "EWY_Korea_zscore_60d", "spx_momentum_3d", "DHR_vol_20d", "spx_abs_ret_max_5d", "SLB_Schlumberger_ret_5d", "CMCSA_ret_1d", "IYM_BasicMaterials_ret_20d", "3M_ret_5d", "EWG_Germany_ret_20d", "PAYX_Paychex_vol_20d"], "is_new": true}, {"model_id": "new_h2_CALM_GradientBoosting_N20_t7", "algo": "GradientBoosting", "regime": "CALM", "horizon": 2, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "WTI_Oil_FRED_zscore_60d", "EFFR_ret_1d", "NFCI_ret_5d", "gjr_condvar_h1", "HangSeng_HK_ret_1d", "AVB_AvalonBay_zscore_60d", "PAYX_Paychex_vol_20d", "AXP_Amex_vol_20d", "EWG_Germany_vol_20d", "AORD_AUS_zscore_60d", "SO_SouthernCo_ret_5d", "TM_Telephone_vol_20d", "ASX_Australia_ret_5d", "SPY_zscore_60d", "heston_var_ev_h3", "BTI_BritishAmerican_ret_20d", "PCAR_PaccarInc_ret_5d", "EFFR_vol_20d"], "is_new": true}, {"model_id": "new_h2_CALM_GradientBoosting_N25_t0", "algo": "GradientBoosting", "regime": "CALM", "horizon": 2, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "T_ret_1d", "EOG_EOGResources_vol_20d", "hmm_p_stress", "HUM_Humana_ret_5d", "HangSeng_HK_ret_1d", "TED_Spread_vol_20d", "IWM_SmallCap_vol_20d", "AORD_AUS_zscore_60d", "AMGN_Amgen_ret_1d", "PFE_ret_1d", "DIS_vol_20d", "SPY_zscore_60d", "SLB_Schlumberger_ret_1d", "US6M_Rate_ret_20d", "Industrial_Production_zscore_60d", "US3M_Rate_vol_20d", "ES_Evergy_ret_1d", "PLD_Prologis_ret_5d", "GILD_Gilead_ret_20d", "CLX_Clorox_vol_20d", "Brent_Oil_FRED_ret_5d", "NVDA_vol_20d", "TM_Telephone_vol_20d"], "is_new": true}, {"model_id": "new_h2_CALM_GradientBoosting_N25_t1", "algo": "GradientBoosting", "regime": "CALM", "horizon": 2, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "CTAS_Cintas_vol_20d", "EOG_EOGResources_ret_5d", "PPL_PPL_ret_1d", "Michigan_Sentiment_ret_20d", "MS_MorganStanley_ret_5d", "DE_Deere_ret_5d", "AMD_ret_5d", "US1Y_Rate_ret_5d", "ASX_Australia_ret_5d", "TED_Spread_zscore_60d", "DE_Deere_vol_20d", "AMGN_Amgen_ret_1d", "EWQ_France_zscore_60d", "Core_PCE_zscore_60d", "PCAR_PaccarInc_ret_5d", "BDX_Becton_Dickinson_ret_20d", "EOG_EOGResources_vol_20d", "NVDA_vol_20d", "EWA_Australia_zscore_60d", "FedFunds_zscore_60d", "HD_ret_1d", "GD_GeneralDynamics_zscore_60d", "EWL_Switzerland_vol_20d"], "is_new": true}, {"model_id": "new_h2_CALM_GradientBoosting_N25_t2", "algo": "GradientBoosting", "regime": "CALM", "horizon": 2, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "IYR_US_REIT2_zscore_60d", "T10Y2Y_Spread_ret_5d", "SBUX_vol_20d", "3M_vol_20d", "CCI_CrownCastle_vol_20d", "EQIX_Equinix_ret_5d", "AMD_ret_1d", "HUM_Humana_ret_5d", "DHR_vol_20d", "HangSeng_HK_ret_1d", "EMR_Emerson_ret_20d", "MRK_Merck_zscore_60d", "EWL_Switzerland_vol_20d", "NEE_NextEra_ret_20d", "VVIX_ret_20d", "AMGN_Amgen_ret_1d", "US7Y_Rate_ret_20d", "CMCSA_ret_1d", "EFFR_ret_1d", "EWM_Malaysia_ret_1d", "PFE_ret_1d", "LUV_SouthwestAir_ret_5d", "EWY_Korea_ret_20d"], "is_new": true}, {"model_id": "new_h2_CALM_GradientBoosting_N25_t3", "algo": "GradientBoosting", "regime": "CALM", "horizon": 2, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "Michigan_Sentiment_ret_20d", "BLK_BlackRock_zscore_60d", "EWY_Korea_ret_20d", "XLK_Tech_zscore_60d", "MSTR_Bitcoin3_ret_1d", "hmm_p_stress", "PAYX_Paychex_vol_20d", "3M_ret_5d", "DE_Deere_ret_5d", "HUM_Humana_ret_5d", "EWS_Singapore_ret_5d", "spx_abs_ret_max_5d", "INTC_ret_5d", "DE_Deere_vol_20d", "HangSeng_HK_vol_20d", "AVB_AvalonBay_zscore_60d", "LMT_LockheedMartin_vol_20d", "XLF_Fin_vol_20d", "EWL_Switzerland_zscore_60d", "NWL_Newell_ret_20d", "EXC_Exelon_ret_1d", "SJM_JM_Smucker_ret_5d", "CCI_CrownCastle_vol_20d"], "is_new": true}, {"model_id": "new_h2_CALM_GradientBoosting_N25_t4", "algo": "GradientBoosting", "regime": "CALM", "horizon": 2, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "AMGN_Amgen_ret_1d", "EWS_Singapore_ret_5d", "M_Macys_vol_20d", "GE_ret_1d", "SLB_Schlumberger_ret_5d", "TXN_vol_20d", "DIS_vol_20d", "Core_PCE_zscore_60d", "LMT_LockheedMartin_vol_20d", "US3M_Rate_zscore_60d", "PAYX_Paychex_zscore_60d", "SJM_JM_Smucker_ret_1d", "EWY_Korea_ret_20d", "SPY_zscore_60d", "MO_AltriaMG_ret_1d", "GILD_Gilead_ret_20d", "spx_vol_5d", "NFCI_ret_5d", "DOW_Price_zscore_60d", "EWC_Canada_zscore_60d", "DAX_Germany_zscore_60d", "EWM_Malaysia_zscore_60d", "Nikkei_Japan_vol_20d"], "is_new": true}, {"model_id": "new_h2_CALM_GradientBoosting_N25_t5", "algo": "GradientBoosting", "regime": "CALM", "horizon": 2, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "XLB_Materials_zscore_60d", "XOM_ret_1d", "TED_Spread_zscore_60d", "NEE_NextEra_ret_20d", "HD_ret_20d", "US7Y_Rate_ret_20d", "Brent_Oil_FRED_ret_20d", "DE_Deere_ret_5d", "MRK_Merck_zscore_60d", "US6M_Rate_ret_20d", "IBEX_Spain_ret_20d", "US3M_Rate_zscore_60d", "SBUX_vol_20d", "EOG_EOGResources_vol_20d", "DE_Deere_vol_20d", "AXP_Amex_vol_20d", "EXC_Exelon_ret_1d", "ES_Evergy_ret_1d", "TED_Spread_vol_20d", "MS_MorganStanley_ret_1d", "vix_mean_abs_ret_5d", "ENB_EnbridgeInc_ret_1d", "NFCI_ret_5d"], "is_new": true}, {"model_id": "new_h2_CALM_GradientBoosting_N25_t6", "algo": "GradientBoosting", "regime": "CALM", "horizon": 2, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "NOC_Northrop_ret_20d", "MS_MorganStanley_zscore_60d", "Brent_Oil_FRED_ret_20d", "PCAR_PaccarInc_ret_5d", "US3M_Rate_zscore_60d", "US7Y_Rate_ret_20d", "NEE_NextEra_ret_20d", "EQIX_Equinix_ret_5d", "EWA_Australia_ret_1d", "LLY_zscore_60d", "MSTR_Bitcoin3_ret_1d", "XLK_Tech_zscore_60d", "VRP_ma5", "WTI_Oil_FRED_zscore_60d", "EFFR_ret_1d", "VOD_Vodafone_zscore_60d", "XLB_Materials_zscore_60d", "T_ret_1d", "EWY_Korea_ret_20d", "EWL_Switzerland_vol_20d", "ITT_ITTInc_ret_5d", "US1Y_Rate_ret_20d", "Retail_Sales_zscore_60d"], "is_new": true}, {"model_id": "new_h2_CALM_GradientBoosting_N25_t7", "algo": "GradientBoosting", "regime": "CALM", "horizon": 2, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "XLK_Tech_zscore_60d", "ES_Evergy_ret_1d", "MRK_Merck_zscore_60d", "ORCL_vol_20d", "AMD_ret_5d", "HangSeng_HK_ret_5d", "DAX_Germany_vol_20d", "EWY_Korea_ret_20d", "EXC_Exelon_ret_1d", "TED_Spread_vol_20d", "EOG_EOGResources_vol_20d", "AXP_Amex_vol_20d", "T10Y2Y_Spread_ret_5d", "EWY_Korea_zscore_60d", "XLF_Fin_vol_20d", "heston_var_ev_h5", "CPB_CampbellSoup_ret_20d", "IWM_SmallCap_vol_20d", "EWG_Germany_ret_20d", "TED_Spread_zscore_60d", "EWA_Australia_ret_1d", "CI_Cigna_vol_20d", "EFFR_ret_1d"], "is_new": true}, {"model_id": "new_h2_CALM_GradientBoosting_N30_t0", "algo": "GradientBoosting", "regime": "CALM", "horizon": 2, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "3M_vol_20d", "BTI_BritishAmerican_ret_5d", "HD_zscore_60d", "HangSeng_HK_ret_1d", "AXP_Amex_vol_20d", "PLD_Prologis_ret_5d", "M_Macys_vol_20d", "DIS_vol_20d", "XLV_Health_zscore_60d", "vix_acceleration_1d", "HangSeng_HK_vol_20d", "EWM_Malaysia_vol_20d", "ASX_Australia_ret_5d", "HD_ret_20d", "US3M_Rate_vol_20d", "Brent_Oil_FRED_ret_20d", "EWL_Switzerland_vol_20d", "spx_momentum_3d", "Core_PCE_zscore_60d", "AMGN_Amgen_ret_1d", "EWG_Germany_vol_20d", "SPY_zscore_60d", "US5Y_Rate_ret_5d", "SCHW_Schwab_ret_5d", "ITT_ITTInc_ret_5d", "DOW_Price_zscore_60d", "TXN_vol_20d", "PPL_PPL_ret_1d"], "is_new": true}, {"model_id": "new_h2_CALM_GradientBoosting_N30_t1", "algo": "GradientBoosting", "regime": "CALM", "horizon": 2, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "vix_mean_abs_ret_5d", "MSTR_Bitcoin3_ret_5d", "HangSeng_HK_ret_1d", "EFFR_vol_20d", "DE_Deere_vol_20d", "HUM_Humana_ret_5d", "AMD_ret_1d", "IYM_BasicMaterials_ret_20d", "XLY_Disc_vol_20d", "DOW_Price_zscore_60d", "BDX_Becton_Dickinson_ret_20d", "SPY_zscore_60d", "EQR_Equity_ret_1d", "EWM_Malaysia_zscore_60d", "Industrial_Production_zscore_60d", "heston_var_ev_h3", "AORD_AUS_zscore_60d", "US3M_Rate_zscore_60d", "SBUX_vol_20d", "Nikkei_Japan_vol_20d", "VVIX_ret_20d", "CPB_CampbellSoup_vol_20d", "EXC_Exelon_zscore_60d", "US30Y_Rate_ret_20d", "M_Macys_vol_20d", "DAX_Germany_zscore_60d", "XLB_Materials_zscore_60d", "TXN_vol_20d"], "is_new": true}, {"model_id": "new_h2_CALM_GradientBoosting_N30_t2", "algo": "GradientBoosting", "regime": "CALM", "horizon": 2, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "3M_vol_20d", "CI_Cigna_vol_20d", "QQQ_vol_20d", "spx_momentum_3d", "LUV_SouthwestAir_ret_5d", "EWS_Singapore_ret_5d", "XOM_ret_20d", "3M_ret_5d", "PAYX_Paychex_vol_20d", "XLV_Health_zscore_60d", "NEE_NextEra_ret_20d", "gjr_condvar_h1", "CPB_CampbellSoup_ret_5d", "TED_Spread_vol_20d", "Nikkei_Japan_zscore_60d", "US3Y_Rate_ret_5d", "IYR_US_REIT2_zscore_60d", "Retail_Sales_zscore_60d", "EFFR_ret_1d", "HD_zscore_60d", "vix_mean_abs_ret_5d", "IBEX_Spain_ret_20d", "EWY_Korea_ret_20d", "HangSeng_HK_vol_20d", "SBUX_zscore_60d", "EXC_Exelon_ret_1d", "Core_PCE_zscore_60d", "EMR_Emerson_ret_20d"], "is_new": true}, {"model_id": "new_h2_CALM_GradientBoosting_N30_t3", "algo": "GradientBoosting", "regime": "CALM", "horizon": 2, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "SBUX_ret_5d", "spx_vol_5d", "XLY_Disc_vol_20d", "INTC_ret_1d", "Michigan_Sentiment_ret_20d", "CPB_CampbellSoup_ret_20d", "AMD_ret_5d", "BA_ret_1d", "SJM_JM_Smucker_ret_1d", "SBUX_zscore_60d", "SCHW_Schwab_ret_5d", "EWC_Canada_zscore_60d", "EOG_EOGResources_ret_5d", "Brent_Oil_FRED_ret_20d", "heston_var_ev_h3", "LMT_LockheedMartin_ret_1d", "BLK_BlackRock_zscore_60d", "EWM_Malaysia_ret_1d", "NVDA_vol_20d", "CLX_Clorox_vol_20d", "US5Y_Rate_ret_5d", "EQIX_Equinix_ret_5d", "AXP_Amex_ret_20d", "DIS_vol_20d", "IYR_US_REIT2_zscore_60d", "vix_acceleration_1d", "XOM_ret_20d", "SPY_zscore_60d"], "is_new": true}, {"model_id": "new_h2_CALM_GradientBoosting_N30_t4", "algo": "GradientBoosting", "regime": "CALM", "horizon": 2, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "AVB_AvalonBay_zscore_60d", "EWA_Australia_ret_1d", "HangSeng_HK_ret_1d", "ES_Evergy_ret_1d", "HUM_Humana_ret_5d", "IBEX_Spain_ret_20d", "GILD_Gilead_ret_20d", "PAYX_Paychex_vol_20d", "HD_ret_5d", "T_ret_1d", "XLF_Fin_vol_20d", "3M_ret_5d", "GE_ret_1d", "SCHW_Schwab_ret_5d", "Nikkei_Japan_zscore_60d", "AMZN_ret_5d", "PAYX_Paychex_ret_20d", "TGT_Target_zscore_60d", "NOC_Northrop_ret_20d", "DAX_Germany_zscore_60d", "EWM_Malaysia_ret_1d", "INTC_ret_1d", "BTI_BritishAmerican_ret_20d", "3M_vol_20d", "PPL_PPL_ret_1d", "NFCI_ret_5d", "ITT_ITTInc_ret_5d", "Brent_Oil_FRED_ret_5d"], "is_new": true}, {"model_id": "new_h2_CALM_GradientBoosting_N30_t5", "algo": "GradientBoosting", "regime": "CALM", "horizon": 2, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "US1Y_Rate_ret_5d", "LOW_Lowes_ret_20d", "CLX_Clorox_vol_20d", "SBUX_ret_5d", "EWQ_France_zscore_60d", "ENB_EnbridgeInc_ret_1d", "TM_Telephone_ret_1d", "VRP_ma5", "LLY_zscore_60d", "CCI_CrownCastle_vol_20d", "GD_GeneralDynamics_zscore_60d", "LMT_LockheedMartin_vol_20d", "MSTR_Bitcoin3_ret_5d", "EWA_Australia_zscore_60d", "EXC_Exelon_ret_1d", "ASX_Australia_vol_20d", "XLB_Materials_zscore_60d", "Brent_Oil_FRED_ret_20d", "vix_mean_abs_ret_5d", "XOM_ret_20d", "SLB_Schlumberger_ret_5d", "3M_ret_5d", "heston_var_ev_h3", "US7Y_Rate_ret_20d", "TXN_vol_20d", "heston_ev_h3", "CPB_CampbellSoup_ret_20d", "ITT_ITTInc_ret_5d"], "is_new": true}, {"model_id": "new_h2_CALM_GradientBoosting_N30_t6", "algo": "GradientBoosting", "regime": "CALM", "horizon": 2, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "vix_mean_abs_ret_5d", "LUV_SouthwestAir_ret_5d", "EWY_Korea_ret_20d", "XLB_Materials_zscore_60d", "Core_CPI_zscore_60d", "MS_MorganStanley_zscore_60d", "AXP_Amex_vol_20d", "SBUX_zscore_60d", "QQQ_vol_20d", "3M_ret_5d", "heston_var_ev_h7", "HD_ret_20d", "AMD_ret_1d", "VOD_Vodafone_zscore_60d", "EWS_Singapore_ret_5d", "AMD_ret_5d", "IBEX_Spain_ret_20d", "INTC_ret_5d", "XLF_Fin_vol_20d", "Core_PCE_zscore_60d", "TM_Telephone_vol_20d", "GILD_Gilead_ret_20d", "AMGN_Amgen_ret_1d", "spx_vol_5d", "TXN_vol_20d", "DOW_Price_zscore_60d", "EQIX_Equinix_ret_5d", "CCI_CrownCastle_vol_20d"], "is_new": true}, {"model_id": "new_h2_CALM_GradientBoosting_N30_t7", "algo": "GradientBoosting", "regime": "CALM", "horizon": 2, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "MS_MorganStanley_ret_5d", "EMR_Emerson_ret_20d", "AVB_AvalonBay_zscore_60d", "EWQ_France_zscore_60d", "SO_SouthernCo_ret_5d", "MRK_Merck_zscore_60d", "TXN_vol_20d", "US1Y_Rate_ret_20d", "M_Macys_vol_20d", "HD_ret_1d", "XOM_ret_20d", "AMT_AmericanTower_ret_1d", "PAYX_Paychex_ret_20d", "TGT_Target_zscore_60d", "TM_Telephone_vol_20d", "AMD_ret_5d", "DOW_Price_zscore_60d", "vix_mean_abs_ret_5d", "ITT_ITTInc_ret_5d", "EFFR_vol_20d", "Industrial_Production_zscore_60d", "CMCSA_ret_1d", "MSTR_Bitcoin3_ret_20d", "HD_ret_20d", "ORCL_vol_20d", "EWG_Germany_vol_20d", "US3Y_Rate_ret_5d", "EWQ_France_ret_20d"], "is_new": true}, {"model_id": "new_h2_CALM_RandomForest_N5_t0", "algo": "RandomForest", "regime": "CALM", "horizon": 2, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "BDX_Becton_Dickinson_ret_20d", "EWA_Australia_ret_1d", "XLB_Materials_zscore_60d"], "is_new": true}, {"model_id": "new_h2_CALM_RandomForest_N5_t1", "algo": "RandomForest", "regime": "CALM", "horizon": 2, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "SJM_JM_Smucker_ret_5d", "SLB_Schlumberger_ret_1d", "BA_ret_1d"], "is_new": true}, {"model_id": "new_h2_CALM_RandomForest_N5_t2", "algo": "RandomForest", "regime": "CALM", "horizon": 2, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "AVB_AvalonBay_zscore_60d", "VVIX_ret_20d", "EOG_EOGResources_ret_5d"], "is_new": true}, {"model_id": "new_h2_CALM_RandomForest_N5_t3", "algo": "RandomForest", "regime": "CALM", "horizon": 2, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWL_Switzerland_zscore_60d", "CCI_CrownCastle_vol_20d", "CMCSA_ret_1d"], "is_new": true}, {"model_id": "new_h2_CALM_RandomForest_N5_t4", "algo": "RandomForest", "regime": "CALM", "horizon": 2, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "CPB_CampbellSoup_ret_5d", "CPB_CampbellSoup_ret_20d", "IYR_US_REIT2_zscore_60d"], "is_new": true}, {"model_id": "new_h2_CALM_RandomForest_N5_t5", "algo": "RandomForest", "regime": "CALM", "horizon": 2, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "JNJ_ret_1d", "EMR_Emerson_ret_20d", "M_Macys_vol_20d"], "is_new": true}, {"model_id": "new_h2_CALM_RandomForest_N5_t6", "algo": "RandomForest", "regime": "CALM", "horizon": 2, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "VVIX_ret_20d", "spx_abs_ret_max_5d", "Michigan_Sentiment_ret_20d"], "is_new": true}, {"model_id": "new_h2_CALM_RandomForest_N5_t7", "algo": "RandomForest", "regime": "CALM", "horizon": 2, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "DIS_vol_20d", "AMGN_Amgen_ret_1d", "INTC_ret_1d"], "is_new": true}, {"model_id": "new_h2_CALM_RandomForest_N8_t0", "algo": "RandomForest", "regime": "CALM", "horizon": 2, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "Retail_Sales_zscore_60d", "GE_ret_1d", "LUV_SouthwestAir_ret_5d", "HD_ret_1d", "HUM_Humana_ret_5d", "BDX_Becton_Dickinson_ret_20d"], "is_new": true}, {"model_id": "new_h2_CALM_RandomForest_N8_t1", "algo": "RandomForest", "regime": "CALM", "horizon": 2, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "PAYX_Paychex_ret_20d", "BTI_BritishAmerican_ret_20d", "HD_zscore_60d", "MS_MorganStanley_ret_1d", "Brent_Oil_FRED_ret_20d", "XLY_Disc_vol_20d"], "is_new": true}, {"model_id": "new_h2_CALM_RandomForest_N8_t2", "algo": "RandomForest", "regime": "CALM", "horizon": 2, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "NVDA_vol_20d", "SBUX_vol_20d", "MS_MorganStanley_ret_1d", "NEE_NextEra_ret_20d", "DE_Deere_ret_5d", "EQR_Equity_ret_1d"], "is_new": true}, {"model_id": "new_h2_CALM_RandomForest_N8_t3", "algo": "RandomForest", "regime": "CALM", "horizon": 2, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "LUV_SouthwestAir_ret_5d", "EWM_Malaysia_zscore_60d", "AMD_ret_5d", "DHR_ret_1d", "MSTR_Bitcoin3_ret_5d", "Brent_Oil_FRED_ret_20d"], "is_new": true}, {"model_id": "new_h2_CALM_RandomForest_N8_t4", "algo": "RandomForest", "regime": "CALM", "horizon": 2, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "MRK_Merck_zscore_60d", "SBUX_vol_20d", "US1Y_Rate_ret_5d", "PAYX_Paychex_vol_20d", "3M_vol_20d", "EWL_Switzerland_vol_20d"], "is_new": true}, {"model_id": "new_h2_CALM_RandomForest_N8_t5", "algo": "RandomForest", "regime": "CALM", "horizon": 2, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "BA_ret_1d", "MS_MorganStanley_ret_5d", "EWQ_France_ret_20d", "AVB_AvalonBay_zscore_60d", "MS_MorganStanley_zscore_60d", "EWQ_France_zscore_60d"], "is_new": true}, {"model_id": "new_h2_CALM_RandomForest_N8_t6", "algo": "RandomForest", "regime": "CALM", "horizon": 2, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "DE_Deere_ret_5d", "TED_Spread_zscore_60d", "heston_var_ev_h5", "CLX_Clorox_vol_20d", "heston_var_ev_h7", "heston_ev_h3"], "is_new": true}, {"model_id": "new_h2_CALM_RandomForest_N8_t7", "algo": "RandomForest", "regime": "CALM", "horizon": 2, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "AMD_ret_1d", "EWL_Switzerland_vol_20d", "VOD_Vodafone_zscore_60d", "AXP_Amex_ret_20d", "VRP_ma5", "Core_PCE_zscore_60d"], "is_new": true}, {"model_id": "new_h2_CALM_RandomForest_N10_t0", "algo": "RandomForest", "regime": "CALM", "horizon": 2, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "DHR_vol_20d", "LUV_SouthwestAir_ret_5d", "DE_Deere_vol_20d", "AMT_AmericanTower_ret_1d", "XLF_Fin_vol_20d", "FedFunds_zscore_60d", "Industrial_Production_zscore_60d", "EFFR_vol_20d"], "is_new": true}, {"model_id": "new_h2_CALM_RandomForest_N10_t1", "algo": "RandomForest", "regime": "CALM", "horizon": 2, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "US3M_Rate_zscore_60d", "PLD_Prologis_ret_5d", "VVIX_ret_20d", "LUV_SouthwestAir_ret_5d", "EWM_Malaysia_zscore_60d", "CTAS_Cintas_vol_20d", "US5Y_Rate_ret_5d", "hmm_p_stress"], "is_new": true}, {"model_id": "new_h2_CALM_RandomForest_N10_t2", "algo": "RandomForest", "regime": "CALM", "horizon": 2, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "XOM_ret_1d", "TGT_Target_zscore_60d", "Nikkei_Japan_zscore_60d", "AXP_Amex_ret_20d", "DE_Deere_vol_20d", "PG_ret_20d", "LMT_LockheedMartin_ret_1d", "SCHW_Schwab_ret_5d"], "is_new": true}, {"model_id": "new_h2_CALM_RandomForest_N10_t3", "algo": "RandomForest", "regime": "CALM", "horizon": 2, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "gjr_condvar_h1", "VRP_ma5", "DHR_ret_1d", "AXP_Amex_ret_20d", "INTC_ret_5d", "HD_ret_5d", "US1Y_Rate_ret_5d", "XLB_Materials_zscore_60d"], "is_new": true}, {"model_id": "new_h2_CALM_RandomForest_N10_t4", "algo": "RandomForest", "regime": "CALM", "horizon": 2, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "SO_SouthernCo_ret_5d", "CMCSA_ret_1d", "DAX_Germany_vol_20d", "US1Y_Rate_ret_5d", "AMGN_Amgen_ret_1d", "gjr_condvar_h1", "US30Y_Rate_ret_20d", "Core_PCE_zscore_60d"], "is_new": true}, {"model_id": "new_h2_CALM_RandomForest_N10_t5", "algo": "RandomForest", "regime": "CALM", "horizon": 2, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "TED_Spread_zscore_60d", "EWJ_Japan_vol_20d", "HD_ret_20d", "IYM_BasicMaterials_ret_20d", "MS_MorganStanley_zscore_60d", "Michigan_Sentiment_ret_20d", "EFFR_vol_20d", "ENB_EnbridgeInc_ret_1d"], "is_new": true}, {"model_id": "new_h2_CALM_RandomForest_N10_t6", "algo": "RandomForest", "regime": "CALM", "horizon": 2, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "SCHW_Schwab_ret_5d", "XLV_Health_zscore_60d", "BA_ret_1d", "INTC_ret_1d", "NEE_NextEra_ret_20d", "MSTR_Bitcoin3_ret_20d", "AMD_ret_5d", "EWQ_France_ret_20d"], "is_new": true}, {"model_id": "new_h2_CALM_RandomForest_N10_t7", "algo": "RandomForest", "regime": "CALM", "horizon": 2, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "IBEX_Spain_ret_20d", "TM_Telephone_vol_20d", "NWL_Newell_ret_20d", "AORD_AUS_zscore_60d", "PAYX_Paychex_vol_20d", "AMD_ret_1d", "HUM_Humana_ret_5d", "heston_var_ev_h3"], "is_new": true}, {"model_id": "new_h2_CALM_RandomForest_N12_t0", "algo": "RandomForest", "regime": "CALM", "horizon": 2, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "DE_Deere_ret_5d", "EWC_Canada_zscore_60d", "LMT_LockheedMartin_vol_20d", "BTI_BritishAmerican_ret_20d", "LUV_SouthwestAir_ret_5d", "SLB_Schlumberger_ret_1d", "AMZN_ret_5d", "Brent_Oil_FRED_ret_20d", "MRK_Merck_zscore_60d", "3M_vol_20d"], "is_new": true}, {"model_id": "new_h2_CALM_RandomForest_N12_t1", "algo": "RandomForest", "regime": "CALM", "horizon": 2, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "NWL_Newell_ret_20d", "SJM_JM_Smucker_ret_1d", "CPB_CampbellSoup_ret_20d", "INTC_ret_1d", "BA_ret_1d", "LLY_zscore_60d", "TED_Spread_vol_20d", "ORCL_zscore_60d", "US3Y_Rate_ret_5d", "BLK_BlackRock_zscore_60d"], "is_new": true}, {"model_id": "new_h2_CALM_RandomForest_N12_t2", "algo": "RandomForest", "regime": "CALM", "horizon": 2, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "HangSeng_HK_ret_5d", "Brent_Oil_FRED_ret_5d", "EWH_HongKong_ret_5d", "EWS_Singapore_ret_5d", "VRP_ma5", "IWM_SmallCap_vol_20d", "HangSeng_HK_vol_20d", "PAYX_Paychex_ret_20d", "US3M_Rate_zscore_60d", "FedFunds_zscore_60d"], "is_new": true}, {"model_id": "new_h2_CALM_RandomForest_N12_t3", "algo": "RandomForest", "regime": "CALM", "horizon": 2, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWJ_Japan_vol_20d", "CPB_CampbellSoup_ret_20d", "INTC_ret_5d", "PLD_Prologis_ret_5d", "US7Y_Rate_ret_20d", "BA_ret_1d", "PFE_ret_1d", "DAX_Germany_vol_20d", "US3Y_Rate_ret_5d", "CCI_CrownCastle_vol_20d"], "is_new": true}, {"model_id": "new_h2_CALM_RandomForest_N12_t4", "algo": "RandomForest", "regime": "CALM", "horizon": 2, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "Michigan_Sentiment_ret_20d", "US6M_Rate_ret_20d", "EWJ_Japan_vol_20d", "Nikkei_Japan_vol_20d", "IYM_BasicMaterials_ret_20d", "MS_MorganStanley_ret_1d", "DAX_Germany_vol_20d", "US1Y_Rate_ret_20d", "EWQ_France_ret_20d", "EWA_Australia_zscore_60d"], "is_new": true}, {"model_id": "new_h2_CALM_RandomForest_N12_t5", "algo": "RandomForest", "regime": "CALM", "horizon": 2, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "MS_MorganStanley_zscore_60d", "T_ret_1d", "3M_vol_20d", "MRK_Merck_zscore_60d", "PLD_Prologis_ret_5d", "AXP_Amex_vol_20d", "ES_Evergy_ret_1d", "EWS_Singapore_ret_5d", "Retail_Sales_zscore_60d", "QQQ_vol_20d"], "is_new": true}, {"model_id": "new_h2_CALM_RandomForest_N12_t6", "algo": "RandomForest", "regime": "CALM", "horizon": 2, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "3M_vol_20d", "heston_var_ev_h5", "CPB_CampbellSoup_ret_5d", "US3M_Rate_vol_20d", "ES_Evergy_ret_1d", "EMR_Emerson_ret_20d", "AMZN_ret_5d", "XLB_Materials_zscore_60d", "LOW_Lowes_ret_5d", "Nikkei_Japan_zscore_60d"], "is_new": true}, {"model_id": "new_h2_CALM_RandomForest_N12_t7", "algo": "RandomForest", "regime": "CALM", "horizon": 2, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "DHR_vol_20d", "XLB_Materials_zscore_60d", "Retail_Sales_zscore_60d", "EWJ_Japan_vol_20d", "DE_Deere_vol_20d", "EWH_HongKong_ret_5d", "US3M_Rate_vol_20d", "Michigan_Sentiment_ret_20d", "HD_zscore_60d", "EXC_Exelon_zscore_60d"], "is_new": true}, {"model_id": "new_h2_CALM_RandomForest_N15_t0", "algo": "RandomForest", "regime": "CALM", "horizon": 2, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWQ_France_zscore_60d", "EMR_Emerson_ret_20d", "MS_MorganStanley_zscore_60d", "PFE_ret_1d", "MSTR_Bitcoin3_ret_1d", "XLY_Disc_vol_20d", "VRP_ma5", "spx_abs_ret_max_5d", "ES_Evergy_ret_1d", "EXC_Exelon_zscore_60d", "ENB_EnbridgeInc_ret_1d", "ORCL_zscore_60d", "QQQ_vol_20d"], "is_new": true}, {"model_id": "new_h2_CALM_RandomForest_N15_t1", "algo": "RandomForest", "regime": "CALM", "horizon": 2, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "hmm_p_stress", "DHR_vol_20d", "AXP_Amex_ret_20d", "DE_Deere_vol_20d", "EWQ_France_zscore_60d", "EQR_Equity_ret_1d", "AMT_AmericanTower_ret_1d", "US5Y_Rate_ret_5d", "CPB_CampbellSoup_ret_20d", "EWY_Korea_zscore_60d", "3M_ret_5d", "BTI_BritishAmerican_ret_20d", "NWL_Newell_ret_20d"], "is_new": true}, {"model_id": "new_h2_CALM_RandomForest_N15_t2", "algo": "RandomForest", "regime": "CALM", "horizon": 2, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "SPY_zscore_60d", "HD_ret_1d", "HUM_Humana_ret_5d", "DHR_ret_1d", "EWJ_Japan_vol_20d", "HangSeng_HK_ret_1d", "SCHW_Schwab_ret_5d", "spx_momentum_3d", "Brent_Oil_FRED_ret_20d", "XLB_Materials_zscore_60d", "VRP_ma5", "TED_Spread_vol_20d", "DOW_Price_zscore_60d"], "is_new": true}, {"model_id": "new_h2_CALM_RandomForest_N15_t3", "algo": "RandomForest", "regime": "CALM", "horizon": 2, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "ITT_ITTInc_ret_5d", "CI_Cigna_vol_20d", "IYM_BasicMaterials_ret_20d", "GD_GeneralDynamics_zscore_60d", "JNJ_ret_1d", "DAX_Germany_zscore_60d", "MS_MorganStanley_ret_1d", "BDX_Becton_Dickinson_ret_20d", "PAYX_Paychex_ret_20d", "ORCL_vol_20d", "US3M_Rate_zscore_60d", "XLF_Fin_vol_20d", "XOM_ret_1d"], "is_new": true}, {"model_id": "new_h2_CALM_RandomForest_N15_t4", "algo": "RandomForest", "regime": "CALM", "horizon": 2, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "AORD_AUS_zscore_60d", "3M_ret_5d", "MS_MorganStanley_ret_5d", "HD_ret_20d", "EWS_Singapore_ret_5d", "EWJ_Japan_vol_20d", "EFFR_ret_1d", "ES_Evergy_ret_1d", "Nikkei_Japan_zscore_60d", "XOM_ret_20d", "DOW_Price_zscore_60d", "DAX_Germany_zscore_60d", "SCHW_Schwab_ret_5d"], "is_new": true}, {"model_id": "new_h2_CALM_RandomForest_N15_t5", "algo": "RandomForest", "regime": "CALM", "horizon": 2, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWG_Germany_vol_20d", "Brent_Oil_FRED_ret_5d", "NOC_Northrop_ret_20d", "SLB_Schlumberger_ret_1d", "ASX_Australia_ret_5d", "MS_MorganStanley_zscore_60d", "M_Macys_vol_20d", "HangSeng_HK_ret_1d", "Retail_Sales_zscore_60d", "AORD_AUS_zscore_60d", "PAYX_Paychex_ret_20d", "3M_ret_5d", "EWH_HongKong_ret_5d"], "is_new": true}, {"model_id": "new_h2_CALM_RandomForest_N15_t6", "algo": "RandomForest", "regime": "CALM", "horizon": 2, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "MSTR_Bitcoin3_ret_1d", "ASX_Australia_ret_5d", "IYR_US_REIT2_zscore_60d", "XLV_Health_zscore_60d", "INTC_ret_1d", "EOG_EOGResources_ret_5d", "XOM_ret_1d", "SJM_JM_Smucker_ret_5d", "HangSeng_HK_ret_5d", "DE_Deere_vol_20d", "gjr_condvar_h1", "HD_ret_5d", "CI_Cigna_vol_20d"], "is_new": true}, {"model_id": "new_h2_CALM_RandomForest_N15_t7", "algo": "RandomForest", "regime": "CALM", "horizon": 2, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "BA_ret_1d", "DE_Deere_ret_5d", "SCHW_Schwab_ret_5d", "GE_ret_1d", "AMGN_Amgen_ret_1d", "heston_var_ev_h7", "ORCL_vol_20d", "QQQ_vol_20d", "LUV_SouthwestAir_ret_5d", "SLB_Schlumberger_ret_1d", "ENB_EnbridgeInc_ret_1d", "Nikkei_Japan_vol_20d", "PCAR_PaccarInc_ret_5d"], "is_new": true}, {"model_id": "new_h2_CALM_RandomForest_N20_t0", "algo": "RandomForest", "regime": "CALM", "horizon": 2, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "Nikkei_Japan_zscore_60d", "EFFR_vol_20d", "AMD_ret_5d", "AVB_AvalonBay_zscore_60d", "EWS_Singapore_ret_5d", "Michigan_Sentiment_ret_20d", "EWA_Australia_ret_1d", "SLB_Schlumberger_ret_5d", "spx_vol_5d", "LOW_Lowes_ret_20d", "heston_ev_h3", "FedFunds_zscore_60d", "NFCI_ret_5d", "Brent_Oil_FRED_ret_20d", "HD_ret_20d", "US3Y_Rate_ret_5d", "PAYX_Paychex_vol_20d", "GD_GeneralDynamics_zscore_60d"], "is_new": true}, {"model_id": "new_h2_CALM_RandomForest_N20_t1", "algo": "RandomForest", "regime": "CALM", "horizon": 2, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "T10Y2Y_Spread_ret_5d", "EWL_Switzerland_vol_20d", "VOD_Vodafone_zscore_60d", "US3Y_Rate_ret_5d", "SBUX_vol_20d", "US5Y_Rate_ret_5d", "TM_Telephone_vol_20d", "HD_ret_1d", "WTI_Oil_FRED_zscore_60d", "Brent_Oil_FRED_ret_5d", "EXC_Exelon_ret_1d", "SLB_Schlumberger_ret_1d", "Nikkei_Japan_vol_20d", "ASX_Australia_ret_5d", "XLY_Disc_vol_20d", "INTC_ret_1d", "AMD_ret_5d", "HD_ret_20d"], "is_new": true}, {"model_id": "new_h2_CALM_RandomForest_N20_t2", "algo": "RandomForest", "regime": "CALM", "horizon": 2, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "ORCL_vol_20d", "HUM_Humana_ret_5d", "EWL_Switzerland_vol_20d", "DAX_Germany_zscore_60d", "DE_Deere_ret_5d", "EWL_Switzerland_zscore_60d", "VRP_ma5", "EXC_Exelon_ret_1d", "MSTR_Bitcoin3_ret_20d", "CPB_CampbellSoup_ret_5d", "EFFR_vol_20d", "T_ret_1d", "US7Y_Rate_ret_20d", "TGT_Target_zscore_60d", "hmm_p_stress", "Brent_Oil_FRED_ret_5d", "spx_vol_5d", "CPB_CampbellSoup_vol_20d"], "is_new": true}, {"model_id": "new_h2_CALM_RandomForest_N20_t3", "algo": "RandomForest", "regime": "CALM", "horizon": 2, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWY_Korea_zscore_60d", "AMGN_Amgen_ret_1d", "3M_vol_20d", "Brent_Oil_FRED_ret_20d", "EQR_Equity_ret_1d", "PAYX_Paychex_zscore_60d", "ORCL_vol_20d", "EMR_Emerson_ret_20d", "DHR_ret_1d", "EWA_Australia_zscore_60d", "VRP_ma5", "TED_Spread_zscore_60d", "HangSeng_HK_ret_5d", "SJM_JM_Smucker_ret_1d", "HangSeng_HK_ret_1d", "EOG_EOGResources_vol_20d", "heston_var_ev_h3", "JNJ_ret_1d"], "is_new": true}, {"model_id": "new_h2_CALM_RandomForest_N20_t4", "algo": "RandomForest", "regime": "CALM", "horizon": 2, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "SO_SouthernCo_ret_5d", "SJM_JM_Smucker_ret_5d", "TED_Spread_vol_20d", "DHR_vol_20d", "DAX_Germany_zscore_60d", "EWG_Germany_vol_20d", "PAYX_Paychex_zscore_60d", "heston_var_ev_h5", "TXN_vol_20d", "hmm_p_stress", "SBUX_ret_5d", "BDX_Becton_Dickinson_ret_20d", "IYM_BasicMaterials_ret_20d", "spx_momentum_3d", "HD_zscore_60d", "BTI_BritishAmerican_ret_5d", "MS_MorganStanley_ret_1d", "T10Y2Y_Spread_ret_5d"], "is_new": true}, {"model_id": "new_h2_CALM_RandomForest_N20_t5", "algo": "RandomForest", "regime": "CALM", "horizon": 2, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EXC_Exelon_ret_1d", "US5Y_Rate_ret_5d", "heston_var_ev_h5", "MO_AltriaMG_ret_1d", "M_Macys_vol_20d", "HangSeng_HK_vol_20d", "PLD_Prologis_ret_5d", "WTI_Oil_FRED_zscore_60d", "TM_Telephone_ret_1d", "GE_ret_1d", "MS_MorganStanley_ret_5d", "AMZN_ret_5d", "IWM_SmallCap_vol_20d", "SO_SouthernCo_ret_5d", "Nikkei_Japan_vol_20d", "PAYX_Paychex_ret_20d", "XOM_ret_1d", "GD_GeneralDynamics_zscore_60d"], "is_new": true}, {"model_id": "new_h2_CALM_RandomForest_N20_t6", "algo": "RandomForest", "regime": "CALM", "horizon": 2, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "3M_ret_5d", "XLY_Disc_vol_20d", "PAYX_Paychex_ret_20d", "EFFR_ret_1d", "EWG_Germany_ret_20d", "MS_MorganStanley_zscore_60d", "Core_PCE_zscore_60d", "AXP_Amex_vol_20d", "TXN_vol_20d", "BLK_BlackRock_zscore_60d", "US3Y_Rate_ret_5d", "DHR_vol_20d", "NOC_Northrop_ret_20d", "CI_Cigna_vol_20d", "MSTR_Bitcoin3_ret_5d", "M_Macys_vol_20d", "DE_Deere_ret_5d", "3M_vol_20d"], "is_new": true}, {"model_id": "new_h2_CALM_RandomForest_N20_t7", "algo": "RandomForest", "regime": "CALM", "horizon": 2, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "INTC_ret_5d", "PAYX_Paychex_zscore_60d", "SLB_Schlumberger_ret_1d", "NEE_NextEra_ret_20d", "HD_ret_1d", "IYM_BasicMaterials_ret_20d", "SJM_JM_Smucker_ret_1d", "vix_acceleration_1d", "T10Y2Y_Spread_ret_5d", "M_Macys_vol_20d", "ITT_ITTInc_ret_5d", "IWM_SmallCap_vol_20d", "LOW_Lowes_ret_20d", "SCHW_Schwab_ret_5d", "LMT_LockheedMartin_vol_20d", "CPB_CampbellSoup_ret_5d", "EOG_EOGResources_vol_20d", "EXC_Exelon_ret_1d"], "is_new": true}, {"model_id": "new_h2_CALM_RandomForest_N25_t0", "algo": "RandomForest", "regime": "CALM", "horizon": 2, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "3M_vol_20d", "BA_ret_1d", "MSTR_Bitcoin3_ret_5d", "TED_Spread_vol_20d", "PAYX_Paychex_vol_20d", "Industrial_Production_zscore_60d", "T10Y2Y_Spread_ret_5d", "NOC_Northrop_ret_20d", "TED_Spread_zscore_60d", "DE_Deere_ret_5d", "HD_zscore_60d", "BDX_Becton_Dickinson_ret_20d", "LOW_Lowes_ret_20d", "Brent_Oil_FRED_ret_20d", "heston_var_ev_h5", "MSTR_Bitcoin3_ret_20d", "LMT_LockheedMartin_vol_20d", "EXC_Exelon_ret_1d", "IWM_SmallCap_vol_20d", "CPB_CampbellSoup_zscore_60d", "EFFR_ret_1d", "EOG_EOGResources_vol_20d", "XLB_Materials_zscore_60d"], "is_new": true}, {"model_id": "new_h2_CALM_RandomForest_N25_t1", "algo": "RandomForest", "regime": "CALM", "horizon": 2, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "HUM_Humana_ret_5d", "3M_vol_20d", "DHR_vol_20d", "EWG_Germany_ret_20d", "spx_momentum_3d", "TED_Spread_zscore_60d", "NEE_NextEra_ret_20d", "TGT_Target_zscore_60d", "Nikkei_Japan_vol_20d", "AMGN_Amgen_ret_1d", "SO_SouthernCo_ret_5d", "EWM_Malaysia_zscore_60d", "INTC_ret_5d", "PPL_PPL_ret_1d", "AVB_AvalonBay_zscore_60d", "BTI_BritishAmerican_ret_20d", "PAYX_Paychex_zscore_60d", "AMT_AmericanTower_ret_1d", "EWS_Singapore_ret_5d", "heston_var_ev_h7", "EWA_Australia_zscore_60d", "AXP_Amex_ret_20d", "HangSeng_HK_ret_1d"], "is_new": true}, {"model_id": "new_h2_CALM_RandomForest_N25_t2", "algo": "RandomForest", "regime": "CALM", "horizon": 2, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "BLK_BlackRock_zscore_60d", "EOG_EOGResources_vol_20d", "AXP_Amex_ret_20d", "IYR_US_REIT2_zscore_60d", "AXP_Amex_vol_20d", "TM_Telephone_vol_20d", "XLK_Tech_zscore_60d", "TGT_Target_zscore_60d", "EWM_Malaysia_ret_1d", "AORD_AUS_zscore_60d", "XLF_Fin_vol_20d", "HD_ret_1d", "BTI_BritishAmerican_ret_5d", "EWJ_Japan_vol_20d", "GD_GeneralDynamics_zscore_60d", "US6M_Rate_ret_20d", "AMD_ret_5d", "US7Y_Rate_ret_20d", "Nikkei_Japan_zscore_60d", "BTI_BritishAmerican_ret_20d", "QQQ_vol_20d", "SBUX_vol_20d", "US5Y_Rate_ret_5d"], "is_new": true}, {"model_id": "new_h2_CALM_RandomForest_N25_t3", "algo": "RandomForest", "regime": "CALM", "horizon": 2, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EFFR_ret_1d", "US1Y_Rate_ret_5d", "GD_GeneralDynamics_zscore_60d", "CTAS_Cintas_vol_20d", "EWH_HongKong_ret_5d", "LOW_Lowes_ret_5d", "MS_MorganStanley_ret_1d", "XLK_Tech_zscore_60d", "GE_ret_1d", "TED_Spread_vol_20d", "AMD_ret_5d", "HangSeng_HK_ret_1d", "INTC_ret_5d", "AVB_AvalonBay_zscore_60d", "PG_ret_20d", "EWG_Germany_vol_20d", "LMT_LockheedMartin_vol_20d", "MSTR_Bitcoin3_ret_1d", "SBUX_zscore_60d", "XOM_ret_20d", "SLB_Schlumberger_ret_5d", "gjr_condvar_h1", "US3Y_Rate_ret_5d"], "is_new": true}, {"model_id": "new_h2_CALM_RandomForest_N25_t4", "algo": "RandomForest", "regime": "CALM", "horizon": 2, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "AMGN_Amgen_ret_1d", "Industrial_Production_zscore_60d", "US7Y_Rate_ret_20d", "XOM_ret_20d", "T_ret_1d", "AXP_Amex_ret_20d", "DHR_ret_1d", "Nikkei_Japan_zscore_60d", "3M_ret_5d", "EOG_EOGResources_ret_5d", "XLV_Health_zscore_60d", "EWY_Korea_zscore_60d", "GE_ret_1d", "DIS_vol_20d", "hmm_p_stress", "EWM_Malaysia_zscore_60d", "CMCSA_ret_1d", "LMT_LockheedMartin_vol_20d", "TED_Spread_zscore_60d", "heston_var_ev_h7", "IYR_US_REIT2_zscore_60d", "T10Y2Y_Spread_ret_5d", "HD_ret_1d"], "is_new": true}, {"model_id": "new_h2_CALM_RandomForest_N25_t5", "algo": "RandomForest", "regime": "CALM", "horizon": 2, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "IBEX_Spain_ret_20d", "EFFR_ret_1d", "AVB_AvalonBay_zscore_60d", "EWM_Malaysia_vol_20d", "CPB_CampbellSoup_ret_20d", "US3M_Rate_vol_20d", "ORCL_zscore_60d", "PFE_ret_1d", "PPL_PPL_ret_1d", "HD_ret_5d", "TGT_Target_zscore_60d", "ORCL_vol_20d", "heston_var_ev_h5", "LOW_Lowes_ret_5d", "XLK_Tech_zscore_60d", "LUV_SouthwestAir_ret_5d", "Core_CPI_zscore_60d", "vix_mean_abs_ret_5d", "BTI_BritishAmerican_ret_5d", "SO_SouthernCo_ret_5d", "NWL_Newell_ret_20d", "GD_GeneralDynamics_zscore_60d", "T10Y2Y_Spread_ret_5d"], "is_new": true}, {"model_id": "new_h2_CALM_RandomForest_N25_t6", "algo": "RandomForest", "regime": "CALM", "horizon": 2, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "SJM_JM_Smucker_ret_5d", "T10Y2Y_Spread_ret_5d", "DE_Deere_ret_5d", "XLF_Fin_vol_20d", "INTC_ret_5d", "heston_var_ev_h5", "vix_acceleration_1d", "MS_MorganStanley_zscore_60d", "SO_SouthernCo_ret_5d", "GILD_Gilead_ret_20d", "CPB_CampbellSoup_zscore_60d", "US3Y_Rate_ret_5d", "NVDA_vol_20d", "EFFR_vol_20d", "EWS_Singapore_ret_5d", "EWL_Switzerland_zscore_60d", "EOG_EOGResources_ret_5d", "Nikkei_Japan_vol_20d", "AORD_AUS_zscore_60d", "LOW_Lowes_ret_5d", "T_ret_1d", "MSTR_Bitcoin3_ret_1d", "HD_ret_1d"], "is_new": true}, {"model_id": "new_h2_CALM_RandomForest_N25_t7", "algo": "RandomForest", "regime": "CALM", "horizon": 2, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "HD_zscore_60d", "SPY_zscore_60d", "EWH_HongKong_ret_5d", "CPB_CampbellSoup_ret_5d", "PAYX_Paychex_ret_20d", "EWA_Australia_zscore_60d", "3M_ret_5d", "MSTR_Bitcoin3_ret_5d", "Michigan_Sentiment_ret_20d", "EQR_Equity_ret_1d", "VRP_ma5", "EWJ_Japan_vol_20d", "CLX_Clorox_vol_20d", "MRK_Merck_zscore_60d", "CMCSA_ret_1d", "TED_Spread_zscore_60d", "XLV_Health_zscore_60d", "PLD_Prologis_ret_5d", "TM_Telephone_vol_20d", "SJM_JM_Smucker_ret_5d", "DOW_Price_zscore_60d", "HUM_Humana_ret_5d", "DIS_vol_20d"], "is_new": true}, {"model_id": "new_h2_CALM_RandomForest_N30_t0", "algo": "RandomForest", "regime": "CALM", "horizon": 2, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "NOC_Northrop_ret_20d", "XOM_ret_1d", "T_ret_1d", "VRP_ma5", "AMT_AmericanTower_ret_1d", "NVDA_vol_20d", "EFFR_ret_1d", "EWQ_France_ret_20d", "DE_Deere_vol_20d", "PFE_ret_1d", "DHR_vol_20d", "EWL_Switzerland_zscore_60d", "QQQ_vol_20d", "Core_PCE_zscore_60d", "PAYX_Paychex_zscore_60d", "SJM_JM_Smucker_ret_5d", "US6M_Rate_ret_20d", "AMZN_ret_5d", "DOW_Price_zscore_60d", "EOG_EOGResources_ret_5d", "EWM_Malaysia_vol_20d", "PPL_PPL_ret_1d", "US3Y_Rate_ret_5d", "BDX_Becton_Dickinson_ret_20d", "ASX_Australia_vol_20d", "Nikkei_Japan_vol_20d", "INTC_ret_1d", "GE_ret_1d"], "is_new": true}, {"model_id": "new_h2_CALM_RandomForest_N30_t1", "algo": "RandomForest", "regime": "CALM", "horizon": 2, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "PLD_Prologis_ret_5d", "PAYX_Paychex_vol_20d", "US6M_Rate_ret_20d", "ITT_ITTInc_ret_5d", "EXC_Exelon_zscore_60d", "Brent_Oil_FRED_ret_20d", "XOM_ret_20d", "PAYX_Paychex_ret_20d", "NVDA_vol_20d", "SJM_JM_Smucker_ret_1d", "INTC_ret_1d", "gjr_condvar_h1", "EWG_Germany_ret_20d", "VRP_ma5", "TED_Spread_vol_20d", "Brent_Oil_FRED_ret_5d", "TXN_vol_20d", "TM_Telephone_vol_20d", "AVB_AvalonBay_zscore_60d", "PAYX_Paychex_zscore_60d", "EWH_HongKong_ret_5d", "PG_ret_20d", "SO_SouthernCo_ret_5d", "BTI_BritishAmerican_ret_20d", "LMT_LockheedMartin_vol_20d", "spx_vol_5d", "spx_momentum_3d", "SCHW_Schwab_ret_5d"], "is_new": true}, {"model_id": "new_h2_CALM_RandomForest_N30_t2", "algo": "RandomForest", "regime": "CALM", "horizon": 2, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "SLB_Schlumberger_ret_5d", "XLF_Fin_vol_20d", "XOM_ret_20d", "MS_MorganStanley_ret_1d", "ENB_EnbridgeInc_ret_1d", "Michigan_Sentiment_ret_20d", "T_ret_1d", "EWH_HongKong_ret_5d", "SLB_Schlumberger_ret_1d", "FedFunds_zscore_60d", "heston_var_ev_h7", "SO_SouthernCo_ret_5d", "PAYX_Paychex_ret_20d", "NFCI_ret_5d", "EWG_Germany_vol_20d", "TED_Spread_vol_20d", "AMT_AmericanTower_ret_1d", "EFFR_ret_1d", "AMGN_Amgen_ret_1d", "Retail_Sales_zscore_60d", "M_Macys_vol_20d", "CCI_CrownCastle_vol_20d", "DOW_Price_zscore_60d", "EWQ_France_ret_20d", "VRP_ma5", "CPB_CampbellSoup_zscore_60d", "EQR_Equity_ret_1d", "Core_CPI_zscore_60d"], "is_new": true}, {"model_id": "new_h2_CALM_RandomForest_N30_t3", "algo": "RandomForest", "regime": "CALM", "horizon": 2, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "T_ret_1d", "TXN_vol_20d", "CPB_CampbellSoup_ret_5d", "EFFR_ret_1d", "US3M_Rate_zscore_60d", "heston_var_ev_h7", "TM_Telephone_vol_20d", "T10Y2Y_Spread_ret_5d", "MS_MorganStanley_ret_1d", "spx_vol_5d", "SLB_Schlumberger_ret_5d", "DE_Deere_vol_20d", "AMZN_ret_5d", "SBUX_zscore_60d", "3M_vol_20d", "M_Macys_vol_20d", "heston_var_ev_h3", "VVIX_ret_20d", "ASX_Australia_vol_20d", "Core_PCE_zscore_60d", "AXP_Amex_vol_20d", "PCAR_PaccarInc_ret_5d", "DE_Deere_ret_5d", "EWL_Switzerland_zscore_60d", "SBUX_ret_5d", "DIS_vol_20d", "US1Y_Rate_ret_5d", "DHR_vol_20d"], "is_new": true}, {"model_id": "new_h2_CALM_RandomForest_N30_t4", "algo": "RandomForest", "regime": "CALM", "horizon": 2, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "SLB_Schlumberger_ret_5d", "spx_vol_5d", "ORCL_vol_20d", "HangSeng_HK_ret_5d", "XLF_Fin_vol_20d", "PCAR_PaccarInc_ret_5d", "AXP_Amex_ret_20d", "EWS_Singapore_ret_5d", "INTC_ret_1d", "heston_var_ev_h7", "NVDA_vol_20d", "SBUX_zscore_60d", "IWM_SmallCap_vol_20d", "PFE_ret_1d", "vix_mean_abs_ret_5d", "NFCI_ret_5d", "SPY_zscore_60d", "EWM_Malaysia_ret_1d", "EWA_Australia_ret_1d", "Industrial_Production_zscore_60d", "LMT_LockheedMartin_vol_20d", "PAYX_Paychex_vol_20d", "AMD_ret_5d", "SBUX_vol_20d", "PAYX_Paychex_ret_20d", "EWQ_France_ret_20d", "CPB_CampbellSoup_ret_5d", "EOG_EOGResources_ret_5d"], "is_new": true}, {"model_id": "new_h2_CALM_RandomForest_N30_t5", "algo": "RandomForest", "regime": "CALM", "horizon": 2, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "IYM_BasicMaterials_ret_20d", "JNJ_ret_1d", "gjr_condvar_h1", "AVB_AvalonBay_zscore_60d", "INTC_ret_5d", "ORCL_zscore_60d", "ASX_Australia_ret_5d", "LOW_Lowes_ret_20d", "DAX_Germany_zscore_60d", "CMCSA_ret_1d", "TED_Spread_vol_20d", "Retail_Sales_zscore_60d", "XLY_Disc_vol_20d", "XLB_Materials_zscore_60d", "EWL_Switzerland_zscore_60d", "SPY_zscore_60d", "GILD_Gilead_ret_20d", "HD_ret_1d", "FedFunds_zscore_60d", "AXP_Amex_ret_20d", "IBEX_Spain_ret_20d", "MO_AltriaMG_ret_1d", "SLB_Schlumberger_ret_5d", "TM_Telephone_ret_1d", "EWH_HongKong_ret_5d", "CPB_CampbellSoup_vol_20d", "Brent_Oil_FRED_ret_5d", "VRP_ma5"], "is_new": true}, {"model_id": "new_h2_CALM_RandomForest_N30_t6", "algo": "RandomForest", "regime": "CALM", "horizon": 2, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "hmm_p_stress", "CPB_CampbellSoup_vol_20d", "EFFR_ret_1d", "EWM_Malaysia_vol_20d", "heston_var_ev_h7", "ITT_ITTInc_ret_5d", "AXP_Amex_vol_20d", "EWC_Canada_zscore_60d", "M_Macys_vol_20d", "TM_Telephone_ret_1d", "PAYX_Paychex_ret_20d", "Core_CPI_zscore_60d", "IWM_SmallCap_vol_20d", "MS_MorganStanley_ret_5d", "TGT_Target_zscore_60d", "XLB_Materials_zscore_60d", "Brent_Oil_FRED_ret_5d", "ASX_Australia_vol_20d", "US7Y_Rate_ret_20d", "ORCL_zscore_60d", "PAYX_Paychex_zscore_60d", "PAYX_Paychex_vol_20d", "heston_ev_h3", "BA_ret_1d", "DHR_vol_20d", "EWM_Malaysia_zscore_60d", "HangSeng_HK_ret_5d", "GD_GeneralDynamics_zscore_60d"], "is_new": true}, {"model_id": "new_h2_CALM_RandomForest_N30_t7", "algo": "RandomForest", "regime": "CALM", "horizon": 2, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "NFCI_ret_5d", "QQQ_vol_20d", "US5Y_Rate_ret_5d", "HUM_Humana_ret_5d", "AMZN_ret_5d", "EWM_Malaysia_zscore_60d", "NWL_Newell_ret_20d", "Nikkei_Japan_vol_20d", "HangSeng_HK_ret_1d", "AMT_AmericanTower_ret_1d", "BTI_BritishAmerican_ret_20d", "Nikkei_Japan_zscore_60d", "EWL_Switzerland_vol_20d", "DE_Deere_vol_20d", "XLK_Tech_zscore_60d", "INTC_ret_5d", "Retail_Sales_zscore_60d", "XOM_ret_1d", "SJM_JM_Smucker_ret_1d", "SLB_Schlumberger_ret_1d", "EWH_HongKong_ret_5d", "Core_PCE_zscore_60d", "LMT_LockheedMartin_vol_20d", "WTI_Oil_FRED_zscore_60d", "LUV_SouthwestAir_ret_5d", "PAYX_Paychex_zscore_60d", "EMR_Emerson_ret_20d", "DIS_vol_20d"], "is_new": true}, {"model_id": "new_h2_CALM_LogisticRegression_N5_t0", "algo": "LogisticRegression", "regime": "CALM", "horizon": 2, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "SLB_Schlumberger_ret_5d", "MS_MorganStanley_ret_5d", "US30Y_Rate_ret_20d"], "is_new": true}, {"model_id": "new_h2_CALM_LogisticRegression_N5_t1", "algo": "LogisticRegression", "regime": "CALM", "horizon": 2, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "US3M_Rate_vol_20d", "CLX_Clorox_vol_20d", "IYM_BasicMaterials_ret_20d"], "is_new": true}, {"model_id": "new_h2_CALM_LogisticRegression_N5_t2", "algo": "LogisticRegression", "regime": "CALM", "horizon": 2, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWM_Malaysia_vol_20d", "EOG_EOGResources_vol_20d", "HangSeng_HK_ret_1d"], "is_new": true}, {"model_id": "new_h2_CALM_LogisticRegression_N5_t3", "algo": "LogisticRegression", "regime": "CALM", "horizon": 2, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "T_ret_1d", "LLY_zscore_60d", "XOM_ret_20d"], "is_new": true}, {"model_id": "new_h2_CALM_LogisticRegression_N5_t4", "algo": "LogisticRegression", "regime": "CALM", "horizon": 2, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "MS_MorganStanley_ret_1d", "AMZN_ret_5d", "PLD_Prologis_ret_5d"], "is_new": true}, {"model_id": "new_h2_CALM_LogisticRegression_N5_t5", "algo": "LogisticRegression", "regime": "CALM", "horizon": 2, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "PPL_PPL_ret_1d", "EXC_Exelon_ret_1d", "MS_MorganStanley_ret_1d"], "is_new": true}, {"model_id": "new_h2_CALM_LogisticRegression_N5_t6", "algo": "LogisticRegression", "regime": "CALM", "horizon": 2, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "TXN_vol_20d", "HD_ret_20d", "hmm_p_stress"], "is_new": true}, {"model_id": "new_h2_CALM_LogisticRegression_N5_t7", "algo": "LogisticRegression", "regime": "CALM", "horizon": 2, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "NWL_Newell_ret_20d", "BTI_BritishAmerican_ret_20d", "PAYX_Paychex_ret_20d"], "is_new": true}, {"model_id": "new_h2_CALM_LogisticRegression_N8_t0", "algo": "LogisticRegression", "regime": "CALM", "horizon": 2, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "ENB_EnbridgeInc_ret_1d", "Brent_Oil_FRED_ret_5d", "SO_SouthernCo_ret_5d", "vix_acceleration_1d", "CPB_CampbellSoup_ret_20d", "Michigan_Sentiment_ret_20d"], "is_new": true}, {"model_id": "new_h2_CALM_LogisticRegression_N8_t1", "algo": "LogisticRegression", "regime": "CALM", "horizon": 2, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWM_Malaysia_vol_20d", "HangSeng_HK_ret_1d", "GILD_Gilead_ret_20d", "vix_acceleration_1d", "Retail_Sales_zscore_60d", "MSTR_Bitcoin3_ret_5d"], "is_new": true}, {"model_id": "new_h2_CALM_LogisticRegression_N8_t2", "algo": "LogisticRegression", "regime": "CALM", "horizon": 2, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "SBUX_vol_20d", "PAYX_Paychex_zscore_60d", "HD_ret_5d", "AMZN_ret_5d", "DAX_Germany_zscore_60d", "VRP_ma5"], "is_new": true}, {"model_id": "new_h2_CALM_LogisticRegression_N8_t3", "algo": "LogisticRegression", "regime": "CALM", "horizon": 2, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "vix_acceleration_1d", "heston_var_ev_h3", "EWQ_France_zscore_60d", "EWG_Germany_ret_20d", "AORD_AUS_zscore_60d", "DOW_Price_zscore_60d"], "is_new": true}, {"model_id": "new_h2_CALM_LogisticRegression_N8_t4", "algo": "LogisticRegression", "regime": "CALM", "horizon": 2, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "ES_Evergy_ret_1d", "Core_CPI_zscore_60d", "US1Y_Rate_ret_5d", "EWA_Australia_ret_1d", "NOC_Northrop_ret_20d", "MSTR_Bitcoin3_ret_1d"], "is_new": true}, {"model_id": "new_h2_CALM_LogisticRegression_N8_t5", "algo": "LogisticRegression", "regime": "CALM", "horizon": 2, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "NOC_Northrop_ret_20d", "NFCI_ret_5d", "ORCL_vol_20d", "Brent_Oil_FRED_ret_5d", "3M_ret_5d", "DHR_ret_1d"], "is_new": true}, {"model_id": "new_h2_CALM_LogisticRegression_N8_t6", "algo": "LogisticRegression", "regime": "CALM", "horizon": 2, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EOG_EOGResources_vol_20d", "ASX_Australia_ret_5d", "DE_Deere_vol_20d", "DAX_Germany_zscore_60d", "EWL_Switzerland_zscore_60d", "GD_GeneralDynamics_zscore_60d"], "is_new": true}, {"model_id": "new_h2_CALM_LogisticRegression_N8_t7", "algo": "LogisticRegression", "regime": "CALM", "horizon": 2, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "ORCL_zscore_60d", "NWL_Newell_ret_20d", "US1Y_Rate_ret_20d", "EWL_Switzerland_zscore_60d", "NVDA_vol_20d", "SBUX_vol_20d"], "is_new": true}, {"model_id": "new_h2_CALM_LogisticRegression_N10_t0", "algo": "LogisticRegression", "regime": "CALM", "horizon": 2, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "PAYX_Paychex_zscore_60d", "XLV_Health_zscore_60d", "Brent_Oil_FRED_ret_20d", "IWM_SmallCap_vol_20d", "DOW_Price_zscore_60d", "heston_ev_h3", "HangSeng_HK_ret_5d", "GILD_Gilead_ret_20d"], "is_new": true}, {"model_id": "new_h2_CALM_LogisticRegression_N10_t1", "algo": "LogisticRegression", "regime": "CALM", "horizon": 2, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "AMZN_ret_5d", "XLV_Health_zscore_60d", "ENB_EnbridgeInc_ret_1d", "EWJ_Japan_vol_20d", "EWS_Singapore_ret_5d", "LMT_LockheedMartin_ret_1d", "BTI_BritishAmerican_ret_20d", "PFE_ret_1d"], "is_new": true}, {"model_id": "new_h2_CALM_LogisticRegression_N10_t2", "algo": "LogisticRegression", "regime": "CALM", "horizon": 2, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "HangSeng_HK_vol_20d", "DHR_ret_1d", "MSTR_Bitcoin3_ret_1d", "BA_ret_1d", "CTAS_Cintas_vol_20d", "T_ret_1d", "MSTR_Bitcoin3_ret_5d", "EWY_Korea_ret_20d"], "is_new": true}, {"model_id": "new_h2_CALM_LogisticRegression_N10_t3", "algo": "LogisticRegression", "regime": "CALM", "horizon": 2, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "BTI_BritishAmerican_ret_5d", "US3M_Rate_zscore_60d", "US7Y_Rate_ret_20d", "LOW_Lowes_ret_20d", "heston_var_ev_h5", "MSTR_Bitcoin3_ret_20d", "XLK_Tech_zscore_60d", "heston_ev_h3"], "is_new": true}, {"model_id": "new_h2_CALM_LogisticRegression_N10_t4", "algo": "LogisticRegression", "regime": "CALM", "horizon": 2, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "SBUX_vol_20d", "ORCL_zscore_60d", "MSTR_Bitcoin3_ret_1d", "FedFunds_zscore_60d", "CMCSA_ret_1d", "HangSeng_HK_ret_1d", "XLF_Fin_vol_20d", "AXP_Amex_ret_20d"], "is_new": true}, {"model_id": "new_h2_CALM_LogisticRegression_N10_t5", "algo": "LogisticRegression", "regime": "CALM", "horizon": 2, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EQIX_Equinix_ret_5d", "IYR_US_REIT2_zscore_60d", "JNJ_ret_1d", "BTI_BritishAmerican_ret_20d", "EWH_HongKong_ret_5d", "GE_ret_1d", "HD_ret_5d", "EXC_Exelon_ret_1d"], "is_new": true}, {"model_id": "new_h2_CALM_LogisticRegression_N10_t6", "algo": "LogisticRegression", "regime": "CALM", "horizon": 2, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EQR_Equity_ret_1d", "TGT_Target_zscore_60d", "TM_Telephone_ret_1d", "EWM_Malaysia_vol_20d", "US3Y_Rate_ret_5d", "3M_ret_5d", "HangSeng_HK_ret_1d", "PPL_PPL_ret_1d"], "is_new": true}, {"model_id": "new_h2_CALM_LogisticRegression_N10_t7", "algo": "LogisticRegression", "regime": "CALM", "horizon": 2, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "LOW_Lowes_ret_5d", "GILD_Gilead_ret_20d", "MSTR_Bitcoin3_ret_1d", "CI_Cigna_vol_20d", "MRK_Merck_zscore_60d", "SBUX_ret_5d", "HD_zscore_60d", "US6M_Rate_ret_20d"], "is_new": true}, {"model_id": "new_h2_CALM_LogisticRegression_N12_t0", "algo": "LogisticRegression", "regime": "CALM", "horizon": 2, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "Core_CPI_zscore_60d", "EXC_Exelon_zscore_60d", "VOD_Vodafone_zscore_60d", "PFE_ret_1d", "TED_Spread_vol_20d", "SPY_zscore_60d", "PLD_Prologis_ret_5d", "INTC_ret_5d", "EWJ_Japan_vol_20d", "BTI_BritishAmerican_ret_20d"], "is_new": true}, {"model_id": "new_h2_CALM_LogisticRegression_N12_t1", "algo": "LogisticRegression", "regime": "CALM", "horizon": 2, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "US7Y_Rate_ret_20d", "ITT_ITTInc_ret_5d", "HD_ret_5d", "EWS_Singapore_ret_5d", "ORCL_zscore_60d", "SCHW_Schwab_ret_5d", "hmm_p_stress", "ORCL_vol_20d", "NWL_Newell_ret_20d", "CPB_CampbellSoup_ret_20d"], "is_new": true}, {"model_id": "new_h2_CALM_LogisticRegression_N12_t2", "algo": "LogisticRegression", "regime": "CALM", "horizon": 2, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "SJM_JM_Smucker_ret_5d", "BA_ret_1d", "BLK_BlackRock_zscore_60d", "CPB_CampbellSoup_ret_5d", "TM_Telephone_ret_1d", "Retail_Sales_zscore_60d", "EWS_Singapore_ret_5d", "CI_Cigna_vol_20d", "IBEX_Spain_ret_20d", "EWG_Germany_vol_20d"], "is_new": true}, {"model_id": "new_h2_CALM_LogisticRegression_N12_t3", "algo": "LogisticRegression", "regime": "CALM", "horizon": 2, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "BTI_BritishAmerican_ret_20d", "EWY_Korea_zscore_60d", "EWM_Malaysia_vol_20d", "Nikkei_Japan_zscore_60d", "SBUX_ret_5d", "TGT_Target_zscore_60d", "NFCI_ret_5d", "EOG_EOGResources_ret_5d", "gjr_condvar_h1", "CPB_CampbellSoup_vol_20d"], "is_new": true}, {"model_id": "new_h2_CALM_LogisticRegression_N12_t4", "algo": "LogisticRegression", "regime": "CALM", "horizon": 2, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "CPB_CampbellSoup_ret_20d", "AXP_Amex_ret_20d", "XLV_Health_zscore_60d", "vix_mean_abs_ret_5d", "Michigan_Sentiment_ret_20d", "gjr_condvar_h1", "TED_Spread_vol_20d", "INTC_ret_5d", "NWL_Newell_ret_20d", "HangSeng_HK_ret_5d"], "is_new": true}, {"model_id": "new_h2_CALM_LogisticRegression_N12_t5", "algo": "LogisticRegression", "regime": "CALM", "horizon": 2, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "NEE_NextEra_ret_20d", "heston_var_ev_h5", "T_ret_1d", "hmm_p_stress", "3M_ret_5d", "DHR_vol_20d", "Nikkei_Japan_vol_20d", "US6M_Rate_ret_20d", "spx_momentum_3d", "EWC_Canada_zscore_60d"], "is_new": true}, {"model_id": "new_h2_CALM_LogisticRegression_N12_t6", "algo": "LogisticRegression", "regime": "CALM", "horizon": 2, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EQIX_Equinix_ret_5d", "EWC_Canada_zscore_60d", "AMD_ret_5d", "Michigan_Sentiment_ret_20d", "ASX_Australia_ret_5d", "BA_ret_1d", "ES_Evergy_ret_1d", "EWA_Australia_zscore_60d", "SJM_JM_Smucker_ret_5d", "Nikkei_Japan_zscore_60d"], "is_new": true}, {"model_id": "new_h2_CALM_LogisticRegression_N12_t7", "algo": "LogisticRegression", "regime": "CALM", "horizon": 2, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "DHR_ret_1d", "EOG_EOGResources_ret_5d", "HD_ret_5d", "CPB_CampbellSoup_ret_20d", "HD_zscore_60d", "EFFR_ret_1d", "AMZN_ret_5d", "Michigan_Sentiment_ret_20d", "NEE_NextEra_ret_20d", "Retail_Sales_zscore_60d"], "is_new": true}, {"model_id": "new_h2_CALM_LogisticRegression_N15_t0", "algo": "LogisticRegression", "regime": "CALM", "horizon": 2, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EXC_Exelon_ret_1d", "INTC_ret_5d", "VOD_Vodafone_zscore_60d", "US3M_Rate_vol_20d", "DHR_vol_20d", "AMGN_Amgen_ret_1d", "CPB_CampbellSoup_ret_20d", "M_Macys_vol_20d", "NWL_Newell_ret_20d", "EWY_Korea_ret_20d", "EWG_Germany_vol_20d", "SPY_zscore_60d", "DAX_Germany_zscore_60d"], "is_new": true}, {"model_id": "new_h2_CALM_LogisticRegression_N15_t1", "algo": "LogisticRegression", "regime": "CALM", "horizon": 2, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "ASX_Australia_ret_5d", "INTC_ret_5d", "EWM_Malaysia_zscore_60d", "VOD_Vodafone_zscore_60d", "DE_Deere_ret_5d", "ES_Evergy_ret_1d", "vix_acceleration_1d", "EQIX_Equinix_ret_5d", "AXP_Amex_vol_20d", "MRK_Merck_zscore_60d", "MSTR_Bitcoin3_ret_1d", "DHR_ret_1d", "spx_abs_ret_max_5d"], "is_new": true}, {"model_id": "new_h2_CALM_LogisticRegression_N15_t2", "algo": "LogisticRegression", "regime": "CALM", "horizon": 2, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "Core_CPI_zscore_60d", "EWH_HongKong_ret_5d", "EXC_Exelon_ret_1d", "TGT_Target_zscore_60d", "HangSeng_HK_vol_20d", "EWM_Malaysia_ret_1d", "US5Y_Rate_ret_5d", "CPB_CampbellSoup_ret_5d", "EWA_Australia_zscore_60d", "EWM_Malaysia_vol_20d", "heston_var_ev_h5", "EOG_EOGResources_vol_20d", "LUV_SouthwestAir_ret_5d"], "is_new": true}, {"model_id": "new_h2_CALM_LogisticRegression_N15_t3", "algo": "LogisticRegression", "regime": "CALM", "horizon": 2, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EFFR_vol_20d", "EWS_Singapore_ret_5d", "WTI_Oil_FRED_zscore_60d", "EXC_Exelon_ret_1d", "IYR_US_REIT2_zscore_60d", "NFCI_ret_5d", "HD_ret_20d", "DAX_Germany_zscore_60d", "EWL_Switzerland_vol_20d", "HangSeng_HK_ret_5d", "ES_Evergy_ret_1d", "SJM_JM_Smucker_ret_5d", "M_Macys_vol_20d"], "is_new": true}, {"model_id": "new_h2_CALM_LogisticRegression_N15_t4", "algo": "LogisticRegression", "regime": "CALM", "horizon": 2, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "3M_vol_20d", "EOG_EOGResources_vol_20d", "FedFunds_zscore_60d", "US7Y_Rate_ret_20d", "SO_SouthernCo_ret_5d", "Nikkei_Japan_zscore_60d", "Nikkei_Japan_vol_20d", "EWA_Australia_zscore_60d", "GD_GeneralDynamics_zscore_60d", "BA_ret_1d", "LOW_Lowes_ret_20d", "EWY_Korea_ret_20d", "ORCL_vol_20d"], "is_new": true}, {"model_id": "new_h2_CALM_LogisticRegression_N15_t5", "algo": "LogisticRegression", "regime": "CALM", "horizon": 2, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "spx_vol_5d", "US3Y_Rate_ret_5d", "T_ret_1d", "heston_var_ev_h3", "XOM_ret_20d", "LOW_Lowes_ret_5d", "EQIX_Equinix_ret_5d", "XOM_ret_1d", "PPL_PPL_ret_1d", "EWM_Malaysia_zscore_60d", "LOW_Lowes_ret_20d", "Michigan_Sentiment_ret_20d", "CMCSA_ret_1d"], "is_new": true}, {"model_id": "new_h2_CALM_LogisticRegression_N15_t6", "algo": "LogisticRegression", "regime": "CALM", "horizon": 2, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "MO_AltriaMG_ret_1d", "IYR_US_REIT2_zscore_60d", "US3M_Rate_vol_20d", "BTI_BritishAmerican_ret_20d", "GD_GeneralDynamics_zscore_60d", "SBUX_vol_20d", "MS_MorganStanley_ret_1d", "EFFR_vol_20d", "EWY_Korea_ret_20d", "heston_var_ev_h3", "SLB_Schlumberger_ret_5d", "BLK_BlackRock_zscore_60d", "XLY_Disc_vol_20d"], "is_new": true}, {"model_id": "new_h2_CALM_LogisticRegression_N15_t7", "algo": "LogisticRegression", "regime": "CALM", "horizon": 2, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "GD_GeneralDynamics_zscore_60d", "QQQ_vol_20d", "MO_AltriaMG_ret_1d", "EWL_Switzerland_zscore_60d", "EWY_Korea_ret_20d", "Michigan_Sentiment_ret_20d", "US6M_Rate_ret_20d", "EFFR_vol_20d", "HD_ret_5d", "PAYX_Paychex_ret_20d", "GE_ret_1d", "MS_MorganStanley_ret_5d", "ORCL_vol_20d"], "is_new": true}, {"model_id": "new_h2_CALM_LogisticRegression_N20_t0", "algo": "LogisticRegression", "regime": "CALM", "horizon": 2, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWH_HongKong_ret_5d", "US30Y_Rate_ret_20d", "US6M_Rate_ret_20d", "NEE_NextEra_ret_20d", "DAX_Germany_zscore_60d", "heston_ev_h3", "US5Y_Rate_ret_5d", "US1Y_Rate_ret_20d", "Industrial_Production_zscore_60d", "EWY_Korea_ret_20d", "EFFR_ret_1d", "HD_zscore_60d", "EWG_Germany_ret_20d", "MS_MorganStanley_ret_1d", "SCHW_Schwab_ret_5d", "VRP_ma5", "DOW_Price_zscore_60d", "HangSeng_HK_ret_1d"], "is_new": true}, {"model_id": "new_h2_CALM_LogisticRegression_N20_t1", "algo": "LogisticRegression", "regime": "CALM", "horizon": 2, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EFFR_vol_20d", "heston_var_ev_h5", "PFE_ret_1d", "heston_var_ev_h7", "NVDA_vol_20d", "EWC_Canada_zscore_60d", "XOM_ret_20d", "T_ret_1d", "EWM_Malaysia_ret_1d", "ITT_ITTInc_ret_5d", "GE_ret_1d", "EWJ_Japan_vol_20d", "spx_momentum_3d", "HD_ret_5d", "NFCI_ret_5d", "IYM_BasicMaterials_ret_20d", "SJM_JM_Smucker_ret_5d", "Nikkei_Japan_vol_20d"], "is_new": true}, {"model_id": "new_h2_CALM_LogisticRegression_N20_t2", "algo": "LogisticRegression", "regime": "CALM", "horizon": 2, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "MS_MorganStanley_ret_5d", "TM_Telephone_ret_1d", "DIS_vol_20d", "HUM_Humana_ret_5d", "US3M_Rate_vol_20d", "PPL_PPL_ret_1d", "VRP_ma5", "heston_var_ev_h3", "EWQ_France_ret_20d", "XLV_Health_zscore_60d", "XLF_Fin_vol_20d", "TXN_vol_20d", "spx_momentum_3d", "Brent_Oil_FRED_ret_5d", "NOC_Northrop_ret_20d", "CPB_CampbellSoup_ret_20d", "LMT_LockheedMartin_vol_20d", "EWM_Malaysia_ret_1d"], "is_new": true}, {"model_id": "new_h2_CALM_LogisticRegression_N20_t3", "algo": "LogisticRegression", "regime": "CALM", "horizon": 2, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "SO_SouthernCo_ret_5d", "spx_abs_ret_max_5d", "EWC_Canada_zscore_60d", "PG_ret_20d", "CLX_Clorox_vol_20d", "AORD_AUS_zscore_60d", "EWS_Singapore_ret_5d", "AMT_AmericanTower_ret_1d", "SJM_JM_Smucker_ret_1d", "EWH_HongKong_ret_5d", "QQQ_vol_20d", "DE_Deere_ret_5d", "MSTR_Bitcoin3_ret_20d", "BLK_BlackRock_zscore_60d", "PPL_PPL_ret_1d", "ENB_EnbridgeInc_ret_1d", "3M_vol_20d", "CPB_CampbellSoup_ret_20d"], "is_new": true}, {"model_id": "new_h2_CALM_LogisticRegression_N20_t4", "algo": "LogisticRegression", "regime": "CALM", "horizon": 2, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "US1Y_Rate_ret_5d", "DHR_ret_1d", "MSTR_Bitcoin3_ret_5d", "GILD_Gilead_ret_20d", "US3Y_Rate_ret_5d", "EWH_HongKong_ret_5d", "Core_PCE_zscore_60d", "DE_Deere_ret_5d", "DAX_Germany_vol_20d", "HD_ret_5d", "MS_MorganStanley_zscore_60d", "EWC_Canada_zscore_60d", "VRP_ma5", "HangSeng_HK_ret_1d", "PAYX_Paychex_ret_20d", "ES_Evergy_ret_1d", "ITT_ITTInc_ret_5d", "heston_var_ev_h5"], "is_new": true}, {"model_id": "new_h2_CALM_LogisticRegression_N20_t5", "algo": "LogisticRegression", "regime": "CALM", "horizon": 2, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "HUM_Humana_ret_5d", "LLY_zscore_60d", "Michigan_Sentiment_ret_20d", "EWG_Germany_vol_20d", "EWA_Australia_ret_1d", "WTI_Oil_FRED_zscore_60d", "CPB_CampbellSoup_zscore_60d", "PFE_ret_1d", "DHR_ret_1d", "US7Y_Rate_ret_20d", "VOD_Vodafone_zscore_60d", "gjr_condvar_h1", "HangSeng_HK_ret_1d", "AMD_ret_1d", "EWL_Switzerland_vol_20d", "MS_MorganStanley_zscore_60d", "US3Y_Rate_ret_5d", "FedFunds_zscore_60d"], "is_new": true}, {"model_id": "new_h2_CALM_LogisticRegression_N20_t6", "algo": "LogisticRegression", "regime": "CALM", "horizon": 2, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "heston_var_ev_h5", "NFCI_ret_5d", "MSTR_Bitcoin3_ret_5d", "HD_zscore_60d", "US3M_Rate_zscore_60d", "PFE_ret_1d", "CLX_Clorox_vol_20d", "XLY_Disc_vol_20d", "NEE_NextEra_ret_20d", "PAYX_Paychex_zscore_60d", "AMZN_ret_5d", "T_ret_1d", "EWM_Malaysia_ret_1d", "EWL_Switzerland_vol_20d", "IYM_BasicMaterials_ret_20d", "EWL_Switzerland_zscore_60d", "WTI_Oil_FRED_zscore_60d", "CTAS_Cintas_vol_20d"], "is_new": true}, {"model_id": "new_h2_CALM_LogisticRegression_N20_t7", "algo": "LogisticRegression", "regime": "CALM", "horizon": 2, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EOG_EOGResources_ret_5d", "LMT_LockheedMartin_ret_1d", "US3Y_Rate_ret_5d", "AORD_AUS_zscore_60d", "Core_CPI_zscore_60d", "M_Macys_vol_20d", "EWY_Korea_ret_20d", "US3M_Rate_zscore_60d", "XLB_Materials_zscore_60d", "GE_ret_1d", "US7Y_Rate_ret_20d", "ASX_Australia_ret_5d", "CCI_CrownCastle_vol_20d", "EWA_Australia_zscore_60d", "EWM_Malaysia_ret_1d", "EWM_Malaysia_zscore_60d", "EWL_Switzerland_vol_20d", "QQQ_vol_20d"], "is_new": true}, {"model_id": "new_h2_CALM_LogisticRegression_N25_t0", "algo": "LogisticRegression", "regime": "CALM", "horizon": 2, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "GILD_Gilead_ret_20d", "TGT_Target_zscore_60d", "CPB_CampbellSoup_ret_5d", "HD_ret_20d", "AXP_Amex_ret_20d", "US3M_Rate_vol_20d", "TXN_vol_20d", "FedFunds_zscore_60d", "CLX_Clorox_vol_20d", "BLK_BlackRock_zscore_60d", "Brent_Oil_FRED_ret_20d", "ORCL_vol_20d", "SCHW_Schwab_ret_5d", "WTI_Oil_FRED_zscore_60d", "EWL_Switzerland_vol_20d", "ASX_Australia_vol_20d", "gjr_condvar_h1", "EWY_Korea_zscore_60d", "heston_var_ev_h7", "CMCSA_ret_1d", "CPB_CampbellSoup_vol_20d", "M_Macys_vol_20d", "AMD_ret_1d"], "is_new": true}, {"model_id": "new_h2_CALM_LogisticRegression_N25_t1", "algo": "LogisticRegression", "regime": "CALM", "horizon": 2, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "IBEX_Spain_ret_20d", "spx_abs_ret_max_5d", "EMR_Emerson_ret_20d", "ASX_Australia_vol_20d", "DOW_Price_zscore_60d", "TED_Spread_zscore_60d", "US30Y_Rate_ret_20d", "NOC_Northrop_ret_20d", "heston_var_ev_h5", "Core_PCE_zscore_60d", "Michigan_Sentiment_ret_20d", "TXN_vol_20d", "ASX_Australia_ret_5d", "Brent_Oil_FRED_ret_20d", "XLB_Materials_zscore_60d", "AVB_AvalonBay_zscore_60d", "EWY_Korea_ret_20d", "HD_zscore_60d", "DHR_vol_20d", "DAX_Germany_vol_20d", "EWG_Germany_ret_20d", "EXC_Exelon_zscore_60d", "SJM_JM_Smucker_ret_5d"], "is_new": true}, {"model_id": "new_h2_CALM_LogisticRegression_N25_t2", "algo": "LogisticRegression", "regime": "CALM", "horizon": 2, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EFFR_ret_1d", "BLK_BlackRock_zscore_60d", "EWH_HongKong_ret_5d", "INTC_ret_5d", "EWC_Canada_zscore_60d", "Brent_Oil_FRED_ret_5d", "NWL_Newell_ret_20d", "vix_acceleration_1d", "Retail_Sales_zscore_60d", "US3M_Rate_vol_20d", "CPB_CampbellSoup_ret_20d", "SBUX_vol_20d", "VRP_ma5", "CPB_CampbellSoup_ret_5d", "EWG_Germany_vol_20d", "TXN_vol_20d", "MSTR_Bitcoin3_ret_20d", "heston_var_ev_h5", "CCI_CrownCastle_vol_20d", "ITT_ITTInc_ret_5d", "SBUX_ret_5d", "XLY_Disc_vol_20d", "CLX_Clorox_vol_20d"], "is_new": true}, {"model_id": "new_h2_CALM_LogisticRegression_N25_t3", "algo": "LogisticRegression", "regime": "CALM", "horizon": 2, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "CPB_CampbellSoup_ret_5d", "US7Y_Rate_ret_20d", "IBEX_Spain_ret_20d", "AMGN_Amgen_ret_1d", "DE_Deere_vol_20d", "HangSeng_HK_ret_5d", "PG_ret_20d", "3M_vol_20d", "EMR_Emerson_ret_20d", "T10Y2Y_Spread_ret_5d", "US1Y_Rate_ret_5d", "CLX_Clorox_vol_20d", "HD_zscore_60d", "CMCSA_ret_1d", "Core_PCE_zscore_60d", "DE_Deere_ret_5d", "TGT_Target_zscore_60d", "PFE_ret_1d", "HangSeng_HK_ret_1d", "INTC_ret_5d", "spx_abs_ret_max_5d", "SBUX_vol_20d", "Brent_Oil_FRED_ret_5d"], "is_new": true}, {"model_id": "new_h2_CALM_LogisticRegression_N25_t4", "algo": "LogisticRegression", "regime": "CALM", "horizon": 2, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "SLB_Schlumberger_ret_5d", "MO_AltriaMG_ret_1d", "SBUX_zscore_60d", "MSTR_Bitcoin3_ret_20d", "HUM_Humana_ret_5d", "Nikkei_Japan_vol_20d", "EWA_Australia_ret_1d", "XLB_Materials_zscore_60d", "Brent_Oil_FRED_ret_20d", "EWQ_France_zscore_60d", "US1Y_Rate_ret_20d", "CCI_CrownCastle_vol_20d", "ASX_Australia_vol_20d", "SJM_JM_Smucker_ret_1d", "HangSeng_HK_ret_5d", "CTAS_Cintas_vol_20d", "spx_momentum_3d", "DAX_Germany_vol_20d", "US5Y_Rate_ret_5d", "HD_ret_5d", "EOG_EOGResources_ret_5d", "EWY_Korea_ret_20d", "DOW_Price_zscore_60d"], "is_new": true}, {"model_id": "new_h2_CALM_LogisticRegression_N25_t5", "algo": "LogisticRegression", "regime": "CALM", "horizon": 2, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "Brent_Oil_FRED_ret_5d", "EWQ_France_ret_20d", "CI_Cigna_vol_20d", "IYR_US_REIT2_zscore_60d", "EWL_Switzerland_vol_20d", "IYM_BasicMaterials_ret_20d", "BA_ret_1d", "NFCI_ret_5d", "VVIX_ret_20d", "Nikkei_Japan_zscore_60d", "ES_Evergy_ret_1d", "MS_MorganStanley_ret_1d", "BTI_BritishAmerican_ret_5d", "NEE_NextEra_ret_20d", "T_ret_1d", "EWY_Korea_zscore_60d", "LLY_zscore_60d", "Industrial_Production_zscore_60d", "ENB_EnbridgeInc_ret_1d", "ITT_ITTInc_ret_5d", "EOG_EOGResources_vol_20d", "SJM_JM_Smucker_ret_1d", "IBEX_Spain_ret_20d"], "is_new": true}, {"model_id": "new_h2_CALM_LogisticRegression_N25_t6", "algo": "LogisticRegression", "regime": "CALM", "horizon": 2, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "HD_ret_20d", "SBUX_vol_20d", "BTI_BritishAmerican_ret_20d", "PLD_Prologis_ret_5d", "US7Y_Rate_ret_20d", "TM_Telephone_vol_20d", "VVIX_ret_20d", "XLV_Health_zscore_60d", "GILD_Gilead_ret_20d", "T10Y2Y_Spread_ret_5d", "TED_Spread_vol_20d", "LOW_Lowes_ret_20d", "Nikkei_Japan_vol_20d", "MS_MorganStanley_ret_1d", "XLB_Materials_zscore_60d", "vix_mean_abs_ret_5d", "SLB_Schlumberger_ret_1d", "US3M_Rate_zscore_60d", "WTI_Oil_FRED_zscore_60d", "EWL_Switzerland_zscore_60d", "BDX_Becton_Dickinson_ret_20d", "AORD_AUS_zscore_60d", "MO_AltriaMG_ret_1d"], "is_new": true}, {"model_id": "new_h2_CALM_LogisticRegression_N25_t7", "algo": "LogisticRegression", "regime": "CALM", "horizon": 2, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "LMT_LockheedMartin_ret_1d", "spx_vol_5d", "M_Macys_vol_20d", "EWH_HongKong_ret_5d", "LUV_SouthwestAir_ret_5d", "spx_abs_ret_max_5d", "SLB_Schlumberger_ret_1d", "HUM_Humana_ret_5d", "DIS_vol_20d", "ES_Evergy_ret_1d", "PAYX_Paychex_vol_20d", "CLX_Clorox_vol_20d", "PCAR_PaccarInc_ret_5d", "EFFR_ret_1d", "MS_MorganStanley_ret_1d", "Michigan_Sentiment_ret_20d", "DE_Deere_vol_20d", "LOW_Lowes_ret_5d", "XLK_Tech_zscore_60d", "hmm_p_stress", "AMZN_ret_5d", "EXC_Exelon_zscore_60d", "MSTR_Bitcoin3_ret_20d"], "is_new": true}, {"model_id": "new_h2_CALM_LogisticRegression_N30_t0", "algo": "LogisticRegression", "regime": "CALM", "horizon": 2, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "CI_Cigna_vol_20d", "HUM_Humana_ret_5d", "spx_abs_ret_max_5d", "GILD_Gilead_ret_20d", "CTAS_Cintas_vol_20d", "EWL_Switzerland_zscore_60d", "3M_vol_20d", "MSTR_Bitcoin3_ret_1d", "SLB_Schlumberger_ret_5d", "NVDA_vol_20d", "EQR_Equity_ret_1d", "EXC_Exelon_zscore_60d", "AVB_AvalonBay_zscore_60d", "SO_SouthernCo_ret_5d", "PAYX_Paychex_ret_20d", "CPB_CampbellSoup_vol_20d", "heston_var_ev_h7", "3M_ret_5d", "NFCI_ret_5d", "NEE_NextEra_ret_20d", "TM_Telephone_vol_20d", "BLK_BlackRock_zscore_60d", "GE_ret_1d", "CPB_CampbellSoup_ret_5d", "SBUX_vol_20d", "DE_Deere_vol_20d", "TM_Telephone_ret_1d", "DHR_vol_20d"], "is_new": true}, {"model_id": "new_h2_CALM_LogisticRegression_N30_t1", "algo": "LogisticRegression", "regime": "CALM", "horizon": 2, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWA_Australia_zscore_60d", "SJM_JM_Smucker_ret_5d", "CTAS_Cintas_vol_20d", "DAX_Germany_vol_20d", "LMT_LockheedMartin_ret_1d", "EMR_Emerson_ret_20d", "DHR_ret_1d", "EWM_Malaysia_vol_20d", "DHR_vol_20d", "US1Y_Rate_ret_20d", "DIS_vol_20d", "EQR_Equity_ret_1d", "EOG_EOGResources_vol_20d", "US1Y_Rate_ret_5d", "EFFR_ret_1d", "HD_ret_20d", "AMD_ret_5d", "TM_Telephone_ret_1d", "HD_ret_1d", "PPL_PPL_ret_1d", "IWM_SmallCap_vol_20d", "XLV_Health_zscore_60d", "TED_Spread_vol_20d", "vix_mean_abs_ret_5d", "LOW_Lowes_ret_20d", "HangSeng_HK_ret_1d", "ASX_Australia_ret_5d", "3M_ret_5d"], "is_new": true}, {"model_id": "new_h2_CALM_LogisticRegression_N30_t2", "algo": "LogisticRegression", "regime": "CALM", "horizon": 2, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "PAYX_Paychex_zscore_60d", "XLB_Materials_zscore_60d", "Core_CPI_zscore_60d", "EWM_Malaysia_vol_20d", "MRK_Merck_zscore_60d", "CPB_CampbellSoup_vol_20d", "spx_vol_5d", "EWY_Korea_ret_20d", "LLY_zscore_60d", "AORD_AUS_zscore_60d", "INTC_ret_1d", "EWA_Australia_zscore_60d", "INTC_ret_5d", "PAYX_Paychex_ret_20d", "DIS_vol_20d", "ENB_EnbridgeInc_ret_1d", "TED_Spread_zscore_60d", "VRP_ma5", "DE_Deere_ret_5d", "3M_vol_20d", "ORCL_vol_20d", "HD_zscore_60d", "Core_PCE_zscore_60d", "LOW_Lowes_ret_5d", "SBUX_zscore_60d", "US1Y_Rate_ret_5d", "AVB_AvalonBay_zscore_60d", "PG_ret_20d"], "is_new": true}, {"model_id": "new_h2_CALM_LogisticRegression_N30_t3", "algo": "LogisticRegression", "regime": "CALM", "horizon": 2, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "TM_Telephone_vol_20d", "AMGN_Amgen_ret_1d", "Brent_Oil_FRED_ret_20d", "AMZN_ret_5d", "HangSeng_HK_ret_1d", "CPB_CampbellSoup_vol_20d", "XOM_ret_1d", "HD_ret_5d", "IYM_BasicMaterials_ret_20d", "EWS_Singapore_ret_5d", "HangSeng_HK_ret_5d", "Brent_Oil_FRED_ret_5d", "XLK_Tech_zscore_60d", "IYR_US_REIT2_zscore_60d", "DAX_Germany_zscore_60d", "SLB_Schlumberger_ret_5d", "CPB_CampbellSoup_ret_20d", "FedFunds_zscore_60d", "heston_var_ev_h5", "PPL_PPL_ret_1d", "US1Y_Rate_ret_5d", "NEE_NextEra_ret_20d", "PCAR_PaccarInc_ret_5d", "WTI_Oil_FRED_zscore_60d", "XLV_Health_zscore_60d", "T_ret_1d", "ASX_Australia_vol_20d", "EOG_EOGResources_vol_20d"], "is_new": true}, {"model_id": "new_h2_CALM_LogisticRegression_N30_t4", "algo": "LogisticRegression", "regime": "CALM", "horizon": 2, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWL_Switzerland_zscore_60d", "LLY_zscore_60d", "SJM_JM_Smucker_ret_1d", "US30Y_Rate_ret_20d", "heston_var_ev_h5", "AMZN_ret_5d", "EWM_Malaysia_vol_20d", "PG_ret_20d", "US1Y_Rate_ret_5d", "US7Y_Rate_ret_20d", "Core_CPI_zscore_60d", "DIS_vol_20d", "CPB_CampbellSoup_ret_20d", "vix_mean_abs_ret_5d", "PAYX_Paychex_ret_20d", "EWY_Korea_zscore_60d", "EWQ_France_zscore_60d", "Michigan_Sentiment_ret_20d", "VVIX_ret_20d", "US5Y_Rate_ret_5d", "AMD_ret_5d", "INTC_ret_5d", "IYM_BasicMaterials_ret_20d", "US1Y_Rate_ret_20d", "ITT_ITTInc_ret_5d", "vix_acceleration_1d", "SCHW_Schwab_ret_5d", "MRK_Merck_zscore_60d"], "is_new": true}, {"model_id": "new_h2_CALM_LogisticRegression_N30_t5", "algo": "LogisticRegression", "regime": "CALM", "horizon": 2, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "ES_Evergy_ret_1d", "US5Y_Rate_ret_5d", "3M_ret_5d", "NVDA_vol_20d", "XLB_Materials_zscore_60d", "Nikkei_Japan_zscore_60d", "US3M_Rate_zscore_60d", "heston_var_ev_h3", "AXP_Amex_vol_20d", "HangSeng_HK_ret_1d", "LOW_Lowes_ret_20d", "EFFR_ret_1d", "T10Y2Y_Spread_ret_5d", "XLF_Fin_vol_20d", "TXN_vol_20d", "Nikkei_Japan_vol_20d", "AMD_ret_5d", "MSTR_Bitcoin3_ret_1d", "BTI_BritishAmerican_ret_5d", "heston_var_ev_h5", "MS_MorganStanley_ret_1d", "IYM_BasicMaterials_ret_20d", "spx_momentum_3d", "PFE_ret_1d", "EWQ_France_zscore_60d", "EWM_Malaysia_zscore_60d", "PAYX_Paychex_zscore_60d", "GE_ret_1d"], "is_new": true}, {"model_id": "new_h2_CALM_LogisticRegression_N30_t6", "algo": "LogisticRegression", "regime": "CALM", "horizon": 2, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "M_Macys_vol_20d", "INTC_ret_5d", "DHR_vol_20d", "PAYX_Paychex_ret_20d", "NOC_Northrop_ret_20d", "CMCSA_ret_1d", "EWM_Malaysia_ret_1d", "NVDA_vol_20d", "US30Y_Rate_ret_20d", "BTI_BritishAmerican_ret_5d", "HangSeng_HK_ret_1d", "VOD_Vodafone_zscore_60d", "US1Y_Rate_ret_20d", "BDX_Becton_Dickinson_ret_20d", "DHR_ret_1d", "LOW_Lowes_ret_20d", "EWL_Switzerland_zscore_60d", "AMT_AmericanTower_ret_1d", "SO_SouthernCo_ret_5d", "HD_ret_1d", "EWA_Australia_zscore_60d", "Retail_Sales_zscore_60d", "vix_mean_abs_ret_5d", "EMR_Emerson_ret_20d", "MS_MorganStanley_zscore_60d", "HUM_Humana_ret_5d", "GD_GeneralDynamics_zscore_60d", "HangSeng_HK_ret_5d"], "is_new": true}, {"model_id": "new_h2_CALM_LogisticRegression_N30_t7", "algo": "LogisticRegression", "regime": "CALM", "horizon": 2, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "AMD_ret_1d", "TGT_Target_zscore_60d", "HD_ret_20d", "ES_Evergy_ret_1d", "EWQ_France_zscore_60d", "DHR_vol_20d", "EWY_Korea_zscore_60d", "EOG_EOGResources_vol_20d", "EWJ_Japan_vol_20d", "HangSeng_HK_ret_5d", "SO_SouthernCo_ret_5d", "3M_vol_20d", "ASX_Australia_ret_5d", "GILD_Gilead_ret_20d", "CCI_CrownCastle_vol_20d", "PCAR_PaccarInc_ret_5d", "DAX_Germany_zscore_60d", "HangSeng_HK_vol_20d", "CTAS_Cintas_vol_20d", "IYM_BasicMaterials_ret_20d", "JNJ_ret_1d", "EWY_Korea_ret_20d", "MS_MorganStanley_zscore_60d", "CPB_CampbellSoup_zscore_60d", "IWM_SmallCap_vol_20d", "vix_acceleration_1d", "Industrial_Production_zscore_60d", "INTC_ret_1d"], "is_new": true}, {"model_id": "new_h2_NORMAL_XGBoost_N5_t0", "algo": "XGBoost", "regime": "NORMAL", "horizon": 2, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EOG_EOGResources_vol_20d", "SJM_JM_Smucker_ret_1d", "US3M_Rate_vol_20d"], "is_new": true}, {"model_id": "new_h2_NORMAL_XGBoost_N5_t1", "algo": "XGBoost", "regime": "NORMAL", "horizon": 2, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "ENB_EnbridgeInc_ret_1d", "MO_AltriaMG_ret_1d", "PAYX_Paychex_zscore_60d"], "is_new": true}, {"model_id": "new_h2_NORMAL_XGBoost_N5_t2", "algo": "XGBoost", "regime": "NORMAL", "horizon": 2, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "spx_momentum_3d", "DHR_vol_20d", "XLY_Disc_vol_20d"], "is_new": true}, {"model_id": "new_h2_NORMAL_XGBoost_N5_t3", "algo": "XGBoost", "regime": "NORMAL", "horizon": 2, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWL_Switzerland_zscore_60d", "CI_Cigna_vol_20d", "ITT_ITTInc_ret_5d"], "is_new": true}, {"model_id": "new_h2_NORMAL_XGBoost_N5_t4", "algo": "XGBoost", "regime": "NORMAL", "horizon": 2, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "AXP_Amex_ret_20d", "HD_ret_1d", "AMT_AmericanTower_ret_1d"], "is_new": true}, {"model_id": "new_h2_NORMAL_XGBoost_N5_t5", "algo": "XGBoost", "regime": "NORMAL", "horizon": 2, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "XLY_Disc_vol_20d", "WTI_Oil_FRED_zscore_60d", "CMCSA_ret_1d"], "is_new": true}, {"model_id": "new_h2_NORMAL_XGBoost_N5_t6", "algo": "XGBoost", "regime": "NORMAL", "horizon": 2, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "HUM_Humana_ret_5d", "EFFR_vol_20d", "WTI_Oil_FRED_zscore_60d"], "is_new": true}, {"model_id": "new_h2_NORMAL_XGBoost_N5_t7", "algo": "XGBoost", "regime": "NORMAL", "horizon": 2, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "CLX_Clorox_vol_20d", "NEE_NextEra_ret_20d", "PG_ret_20d"], "is_new": true}, {"model_id": "new_h2_NORMAL_XGBoost_N8_t0", "algo": "XGBoost", "regime": "NORMAL", "horizon": 2, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWQ_France_zscore_60d", "VRP_ma5", "EQIX_Equinix_ret_5d", "Michigan_Sentiment_ret_20d", "MSTR_Bitcoin3_ret_20d", "XLB_Materials_zscore_60d"], "is_new": true}, {"model_id": "new_h2_NORMAL_XGBoost_N8_t1", "algo": "XGBoost", "regime": "NORMAL", "horizon": 2, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWL_Switzerland_zscore_60d", "DAX_Germany_zscore_60d", "LMT_LockheedMartin_vol_20d", "ASX_Australia_ret_5d", "EWL_Switzerland_vol_20d", "DAX_Germany_vol_20d"], "is_new": true}, {"model_id": "new_h2_NORMAL_XGBoost_N8_t2", "algo": "XGBoost", "regime": "NORMAL", "horizon": 2, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "TGT_Target_zscore_60d", "LOW_Lowes_ret_20d", "3M_vol_20d", "ES_Evergy_ret_1d", "DOW_Price_zscore_60d", "AXP_Amex_ret_20d"], "is_new": true}, {"model_id": "new_h2_NORMAL_XGBoost_N8_t3", "algo": "XGBoost", "regime": "NORMAL", "horizon": 2, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "US5Y_Rate_ret_5d", "EWQ_France_zscore_60d", "heston_var_ev_h5", "EWS_Singapore_ret_5d", "EWJ_Japan_vol_20d", "MRK_Merck_zscore_60d"], "is_new": true}, {"model_id": "new_h2_NORMAL_XGBoost_N8_t4", "algo": "XGBoost", "regime": "NORMAL", "horizon": 2, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "CI_Cigna_vol_20d", "XLB_Materials_zscore_60d", "BLK_BlackRock_zscore_60d", "3M_ret_5d", "BDX_Becton_Dickinson_ret_20d", "heston_var_ev_h7"], "is_new": true}, {"model_id": "new_h2_NORMAL_XGBoost_N8_t5", "algo": "XGBoost", "regime": "NORMAL", "horizon": 2, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "CPB_CampbellSoup_ret_5d", "MSTR_Bitcoin3_ret_1d", "DIS_vol_20d", "ES_Evergy_ret_1d", "JNJ_ret_1d", "HD_ret_1d"], "is_new": true}, {"model_id": "new_h2_NORMAL_XGBoost_N8_t6", "algo": "XGBoost", "regime": "NORMAL", "horizon": 2, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "LMT_LockheedMartin_vol_20d", "WTI_Oil_FRED_zscore_60d", "IBEX_Spain_ret_20d", "3M_ret_5d", "PAYX_Paychex_zscore_60d", "US6M_Rate_ret_20d"], "is_new": true}, {"model_id": "new_h2_NORMAL_XGBoost_N8_t7", "algo": "XGBoost", "regime": "NORMAL", "horizon": 2, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "HD_ret_5d", "LLY_zscore_60d", "EWL_Switzerland_vol_20d", "DE_Deere_vol_20d", "EWM_Malaysia_zscore_60d", "CI_Cigna_vol_20d"], "is_new": true}, {"model_id": "new_h2_NORMAL_XGBoost_N10_t0", "algo": "XGBoost", "regime": "NORMAL", "horizon": 2, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "HD_ret_20d", "IYM_BasicMaterials_ret_20d", "MSTR_Bitcoin3_ret_1d", "HD_zscore_60d", "ORCL_zscore_60d", "HD_ret_5d", "EWS_Singapore_ret_5d", "PAYX_Paychex_ret_20d"], "is_new": true}, {"model_id": "new_h2_NORMAL_XGBoost_N10_t1", "algo": "XGBoost", "regime": "NORMAL", "horizon": 2, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWC_Canada_zscore_60d", "XLV_Health_zscore_60d", "CPB_CampbellSoup_vol_20d", "CCI_CrownCastle_vol_20d", "vix_mean_abs_ret_5d", "PAYX_Paychex_zscore_60d", "SJM_JM_Smucker_ret_1d", "Nikkei_Japan_vol_20d"], "is_new": true}, {"model_id": "new_h2_NORMAL_XGBoost_N10_t2", "algo": "XGBoost", "regime": "NORMAL", "horizon": 2, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "LLY_zscore_60d", "EQIX_Equinix_ret_5d", "spx_momentum_3d", "PPL_PPL_ret_1d", "heston_var_ev_h7", "NVDA_vol_20d", "BTI_BritishAmerican_ret_5d", "DAX_Germany_zscore_60d"], "is_new": true}, {"model_id": "new_h2_NORMAL_XGBoost_N10_t3", "algo": "XGBoost", "regime": "NORMAL", "horizon": 2, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "Retail_Sales_zscore_60d", "T_ret_1d", "BDX_Becton_Dickinson_ret_20d", "NEE_NextEra_ret_20d", "spx_vol_5d", "Brent_Oil_FRED_ret_20d", "XOM_ret_1d", "BLK_BlackRock_zscore_60d"], "is_new": true}, {"model_id": "new_h2_NORMAL_XGBoost_N10_t4", "algo": "XGBoost", "regime": "NORMAL", "horizon": 2, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "Core_CPI_zscore_60d", "MS_MorganStanley_ret_5d", "Retail_Sales_zscore_60d", "XOM_ret_20d", "T_ret_1d", "PAYX_Paychex_ret_20d", "AMZN_ret_5d", "MSTR_Bitcoin3_ret_5d"], "is_new": true}, {"model_id": "new_h2_NORMAL_XGBoost_N10_t5", "algo": "XGBoost", "regime": "NORMAL", "horizon": 2, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "IYR_US_REIT2_zscore_60d", "EWS_Singapore_ret_5d", "DE_Deere_vol_20d", "DOW_Price_zscore_60d", "EWC_Canada_zscore_60d", "SBUX_vol_20d", "DAX_Germany_vol_20d", "TM_Telephone_vol_20d"], "is_new": true}, {"model_id": "new_h2_NORMAL_XGBoost_N10_t6", "algo": "XGBoost", "regime": "NORMAL", "horizon": 2, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "MSTR_Bitcoin3_ret_1d", "heston_ev_h3", "SPY_zscore_60d", "HD_ret_20d", "heston_var_ev_h7", "SO_SouthernCo_ret_5d", "MO_AltriaMG_ret_1d", "NOC_Northrop_ret_20d"], "is_new": true}, {"model_id": "new_h2_NORMAL_XGBoost_N10_t7", "algo": "XGBoost", "regime": "NORMAL", "horizon": 2, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "HD_ret_5d", "DOW_Price_zscore_60d", "XLF_Fin_vol_20d", "IBEX_Spain_ret_20d", "EWA_Australia_zscore_60d", "AMT_AmericanTower_ret_1d", "AVB_AvalonBay_zscore_60d", "AORD_AUS_zscore_60d"], "is_new": true}, {"model_id": "new_h2_NORMAL_XGBoost_N12_t0", "algo": "XGBoost", "regime": "NORMAL", "horizon": 2, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWS_Singapore_ret_5d", "US3M_Rate_zscore_60d", "TM_Telephone_ret_1d", "US3M_Rate_vol_20d", "DOW_Price_zscore_60d", "Brent_Oil_FRED_ret_20d", "EWH_HongKong_ret_5d", "DHR_vol_20d", "EXC_Exelon_zscore_60d", "ORCL_vol_20d"], "is_new": true}, {"model_id": "new_h2_NORMAL_XGBoost_N12_t1", "algo": "XGBoost", "regime": "NORMAL", "horizon": 2, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "DAX_Germany_zscore_60d", "Retail_Sales_zscore_60d", "TM_Telephone_ret_1d", "EXC_Exelon_zscore_60d", "CPB_CampbellSoup_vol_20d", "SBUX_zscore_60d", "spx_momentum_3d", "US1Y_Rate_ret_20d", "EWG_Germany_ret_20d", "SJM_JM_Smucker_ret_1d"], "is_new": true}, {"model_id": "new_h2_NORMAL_XGBoost_N12_t2", "algo": "XGBoost", "regime": "NORMAL", "horizon": 2, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWH_HongKong_ret_5d", "TED_Spread_vol_20d", "DHR_ret_1d", "AVB_AvalonBay_zscore_60d", "CPB_CampbellSoup_ret_20d", "CPB_CampbellSoup_ret_5d", "GILD_Gilead_ret_20d", "vix_acceleration_1d", "AORD_AUS_zscore_60d", "MS_MorganStanley_ret_1d"], "is_new": true}, {"model_id": "new_h2_NORMAL_XGBoost_N12_t3", "algo": "XGBoost", "regime": "NORMAL", "horizon": 2, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "HD_zscore_60d", "ES_Evergy_ret_1d", "Nikkei_Japan_zscore_60d", "PAYX_Paychex_ret_20d", "EWL_Switzerland_zscore_60d", "MO_AltriaMG_ret_1d", "BTI_BritishAmerican_ret_20d", "DHR_vol_20d", "SBUX_vol_20d", "EQR_Equity_ret_1d"], "is_new": true}, {"model_id": "new_h2_NORMAL_XGBoost_N12_t4", "algo": "XGBoost", "regime": "NORMAL", "horizon": 2, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWY_Korea_ret_20d", "SBUX_zscore_60d", "LUV_SouthwestAir_ret_5d", "MS_MorganStanley_ret_5d", "INTC_ret_1d", "MSTR_Bitcoin3_ret_5d", "XLV_Health_zscore_60d", "PPL_PPL_ret_1d", "NFCI_ret_5d", "EXC_Exelon_zscore_60d"], "is_new": true}, {"model_id": "new_h2_NORMAL_XGBoost_N12_t5", "algo": "XGBoost", "regime": "NORMAL", "horizon": 2, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EFFR_vol_20d", "ENB_EnbridgeInc_ret_1d", "HD_ret_1d", "US5Y_Rate_ret_5d", "HD_ret_5d", "MO_AltriaMG_ret_1d", "Core_PCE_zscore_60d", "AMT_AmericanTower_ret_1d", "HangSeng_HK_vol_20d", "NEE_NextEra_ret_20d"], "is_new": true}, {"model_id": "new_h2_NORMAL_XGBoost_N12_t6", "algo": "XGBoost", "regime": "NORMAL", "horizon": 2, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "NFCI_ret_5d", "US6M_Rate_ret_20d", "DE_Deere_ret_5d", "US3M_Rate_zscore_60d", "NVDA_vol_20d", "IWM_SmallCap_vol_20d", "MSTR_Bitcoin3_ret_5d", "TGT_Target_zscore_60d", "IYR_US_REIT2_zscore_60d", "CTAS_Cintas_vol_20d"], "is_new": true}, {"model_id": "new_h2_NORMAL_XGBoost_N12_t7", "algo": "XGBoost", "regime": "NORMAL", "horizon": 2, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "PAYX_Paychex_vol_20d", "NOC_Northrop_ret_20d", "SCHW_Schwab_ret_5d", "EQIX_Equinix_ret_5d", "VVIX_ret_20d", "EWM_Malaysia_zscore_60d", "INTC_ret_5d", "AMT_AmericanTower_ret_1d", "XLV_Health_zscore_60d", "ORCL_zscore_60d"], "is_new": true}, {"model_id": "new_h2_NORMAL_XGBoost_N15_t0", "algo": "XGBoost", "regime": "NORMAL", "horizon": 2, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "DHR_ret_1d", "Brent_Oil_FRED_ret_20d", "DE_Deere_ret_5d", "US5Y_Rate_ret_5d", "CTAS_Cintas_vol_20d", "HangSeng_HK_ret_5d", "ORCL_vol_20d", "spx_momentum_3d", "MRK_Merck_zscore_60d", "3M_vol_20d", "ES_Evergy_ret_1d", "DIS_vol_20d", "EWC_Canada_zscore_60d"], "is_new": true}, {"model_id": "new_h2_NORMAL_XGBoost_N15_t1", "algo": "XGBoost", "regime": "NORMAL", "horizon": 2, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "HD_ret_5d", "LLY_zscore_60d", "SCHW_Schwab_ret_5d", "US3M_Rate_zscore_60d", "PG_ret_20d", "DHR_ret_1d", "MRK_Merck_zscore_60d", "Core_CPI_zscore_60d", "T10Y2Y_Spread_ret_5d", "HD_zscore_60d", "spx_abs_ret_max_5d", "BA_ret_1d", "PFE_ret_1d"], "is_new": true}, {"model_id": "new_h2_NORMAL_XGBoost_N15_t2", "algo": "XGBoost", "regime": "NORMAL", "horizon": 2, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "CMCSA_ret_1d", "HD_zscore_60d", "PAYX_Paychex_zscore_60d", "US1Y_Rate_ret_5d", "DHR_vol_20d", "EWS_Singapore_ret_5d", "EWA_Australia_zscore_60d", "PAYX_Paychex_ret_20d", "heston_var_ev_h7", "EFFR_ret_1d", "BDX_Becton_Dickinson_ret_20d", "US3M_Rate_zscore_60d", "GE_ret_1d"], "is_new": true}, {"model_id": "new_h2_NORMAL_XGBoost_N15_t3", "algo": "XGBoost", "regime": "NORMAL", "horizon": 2, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "M_Macys_vol_20d", "PAYX_Paychex_ret_20d", "AMD_ret_5d", "MS_MorganStanley_zscore_60d", "EWC_Canada_zscore_60d", "NEE_NextEra_ret_20d", "gjr_condvar_h1", "EWL_Switzerland_zscore_60d", "vix_mean_abs_ret_5d", "DHR_ret_1d", "T10Y2Y_Spread_ret_5d", "Nikkei_Japan_vol_20d", "EWG_Germany_ret_20d"], "is_new": true}, {"model_id": "new_h2_NORMAL_XGBoost_N15_t4", "algo": "XGBoost", "regime": "NORMAL", "horizon": 2, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "DOW_Price_zscore_60d", "AMZN_ret_5d", "AXP_Amex_vol_20d", "SPY_zscore_60d", "HD_zscore_60d", "SJM_JM_Smucker_ret_5d", "GD_GeneralDynamics_zscore_60d", "DAX_Germany_zscore_60d", "MS_MorganStanley_ret_1d", "EWY_Korea_zscore_60d", "TXN_vol_20d", "EWG_Germany_vol_20d", "EWJ_Japan_vol_20d"], "is_new": true}, {"model_id": "new_h2_NORMAL_XGBoost_N15_t5", "algo": "XGBoost", "regime": "NORMAL", "horizon": 2, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "SLB_Schlumberger_ret_5d", "ENB_EnbridgeInc_ret_1d", "TM_Telephone_vol_20d", "XLB_Materials_zscore_60d", "Michigan_Sentiment_ret_20d", "SJM_JM_Smucker_ret_1d", "heston_var_ev_h3", "CI_Cigna_vol_20d", "EWG_Germany_vol_20d", "MSTR_Bitcoin3_ret_20d", "AMD_ret_1d", "IYR_US_REIT2_zscore_60d", "EWY_Korea_zscore_60d"], "is_new": true}, {"model_id": "new_h2_NORMAL_XGBoost_N15_t6", "algo": "XGBoost", "regime": "NORMAL", "horizon": 2, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "AMZN_ret_5d", "US30Y_Rate_ret_20d", "MO_AltriaMG_ret_1d", "NVDA_vol_20d", "HD_ret_1d", "US3M_Rate_zscore_60d", "BA_ret_1d", "Core_CPI_zscore_60d", "EWA_Australia_ret_1d", "DIS_vol_20d", "EXC_Exelon_zscore_60d", "XOM_ret_1d", "CMCSA_ret_1d"], "is_new": true}, {"model_id": "new_h2_NORMAL_XGBoost_N15_t7", "algo": "XGBoost", "regime": "NORMAL", "horizon": 2, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "spx_abs_ret_max_5d", "3M_ret_5d", "Brent_Oil_FRED_ret_5d", "CPB_CampbellSoup_zscore_60d", "IYM_BasicMaterials_ret_20d", "HD_zscore_60d", "MS_MorganStanley_ret_5d", "CTAS_Cintas_vol_20d", "QQQ_vol_20d", "LLY_zscore_60d", "TED_Spread_zscore_60d", "heston_var_ev_h5", "BLK_BlackRock_zscore_60d"], "is_new": true}, {"model_id": "new_h2_NORMAL_XGBoost_N20_t0", "algo": "XGBoost", "regime": "NORMAL", "horizon": 2, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "HD_ret_5d", "EXC_Exelon_ret_1d", "heston_var_ev_h3", "BTI_BritishAmerican_ret_5d", "BLK_BlackRock_zscore_60d", "HD_ret_20d", "GE_ret_1d", "EWQ_France_zscore_60d", "BTI_BritishAmerican_ret_20d", "LMT_LockheedMartin_ret_1d", "Michigan_Sentiment_ret_20d", "MSTR_Bitcoin3_ret_5d", "SLB_Schlumberger_ret_5d", "JNJ_ret_1d", "Brent_Oil_FRED_ret_5d", "EWJ_Japan_vol_20d", "IYM_BasicMaterials_ret_20d", "EWA_Australia_ret_1d"], "is_new": true}, {"model_id": "new_h2_NORMAL_XGBoost_N20_t1", "algo": "XGBoost", "regime": "NORMAL", "horizon": 2, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "Retail_Sales_zscore_60d", "IBEX_Spain_ret_20d", "PFE_ret_1d", "INTC_ret_5d", "LMT_LockheedMartin_vol_20d", "SLB_Schlumberger_ret_1d", "AXP_Amex_vol_20d", "EWC_Canada_zscore_60d", "AVB_AvalonBay_zscore_60d", "XOM_ret_1d", "heston_var_ev_h3", "AMD_ret_5d", "CPB_CampbellSoup_vol_20d", "Michigan_Sentiment_ret_20d", "SPY_zscore_60d", "MSTR_Bitcoin3_ret_20d", "HD_ret_20d", "PAYX_Paychex_vol_20d"], "is_new": true}, {"model_id": "new_h2_NORMAL_XGBoost_N20_t2", "algo": "XGBoost", "regime": "NORMAL", "horizon": 2, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "ASX_Australia_vol_20d", "CMCSA_ret_1d", "XLF_Fin_vol_20d", "EFFR_ret_1d", "PG_ret_20d", "spx_momentum_3d", "CPB_CampbellSoup_vol_20d", "Michigan_Sentiment_ret_20d", "TED_Spread_vol_20d", "US30Y_Rate_ret_20d", "heston_var_ev_h7", "EWG_Germany_vol_20d", "SBUX_ret_5d", "ORCL_vol_20d", "CPB_CampbellSoup_zscore_60d", "LUV_SouthwestAir_ret_5d", "SPY_zscore_60d", "MS_MorganStanley_ret_1d"], "is_new": true}, {"model_id": "new_h2_NORMAL_XGBoost_N20_t3", "algo": "XGBoost", "regime": "NORMAL", "horizon": 2, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "US3M_Rate_zscore_60d", "US3Y_Rate_ret_5d", "PPL_PPL_ret_1d", "MSTR_Bitcoin3_ret_5d", "ORCL_zscore_60d", "LOW_Lowes_ret_20d", "US6M_Rate_ret_20d", "T10Y2Y_Spread_ret_5d", "EWG_Germany_ret_20d", "spx_vol_5d", "EWC_Canada_zscore_60d", "EFFR_vol_20d", "NFCI_ret_5d", "BTI_BritishAmerican_ret_20d", "QQQ_vol_20d", "AVB_AvalonBay_zscore_60d", "ITT_ITTInc_ret_5d", "AORD_AUS_zscore_60d"], "is_new": true}, {"model_id": "new_h2_NORMAL_XGBoost_N20_t4", "algo": "XGBoost", "regime": "NORMAL", "horizon": 2, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EXC_Exelon_ret_1d", "LOW_Lowes_ret_20d", "heston_var_ev_h5", "TM_Telephone_vol_20d", "BTI_BritishAmerican_ret_5d", "VOD_Vodafone_zscore_60d", "LMT_LockheedMartin_ret_1d", "NWL_Newell_ret_20d", "IWM_SmallCap_vol_20d", "ORCL_zscore_60d", "DOW_Price_zscore_60d", "TXN_vol_20d", "MO_AltriaMG_ret_1d", "EWY_Korea_ret_20d", "LUV_SouthwestAir_ret_5d", "EWQ_France_zscore_60d", "US6M_Rate_ret_20d", "spx_abs_ret_max_5d"], "is_new": true}, {"model_id": "new_h2_NORMAL_XGBoost_N20_t5", "algo": "XGBoost", "regime": "NORMAL", "horizon": 2, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "MS_MorganStanley_ret_1d", "M_Macys_vol_20d", "spx_momentum_3d", "DHR_vol_20d", "CPB_CampbellSoup_ret_20d", "PLD_Prologis_ret_5d", "SBUX_ret_5d", "IYR_US_REIT2_zscore_60d", "US30Y_Rate_ret_20d", "AXP_Amex_ret_20d", "BTI_BritishAmerican_ret_5d", "IYM_BasicMaterials_ret_20d", "TM_Telephone_vol_20d", "GD_GeneralDynamics_zscore_60d", "NWL_Newell_ret_20d", "Nikkei_Japan_vol_20d", "T_ret_1d", "SBUX_vol_20d"], "is_new": true}, {"model_id": "new_h2_NORMAL_XGBoost_N20_t6", "algo": "XGBoost", "regime": "NORMAL", "horizon": 2, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "3M_vol_20d", "DOW_Price_zscore_60d", "SO_SouthernCo_ret_5d", "JNJ_ret_1d", "EWG_Germany_vol_20d", "EFFR_vol_20d", "PG_ret_20d", "EQIX_Equinix_ret_5d", "Retail_Sales_zscore_60d", "MSTR_Bitcoin3_ret_5d", "Nikkei_Japan_zscore_60d", "VRP_ma5", "US30Y_Rate_ret_20d", "heston_var_ev_h7", "EWL_Switzerland_zscore_60d", "EXC_Exelon_ret_1d", "INTC_ret_5d", "BTI_BritishAmerican_ret_5d"], "is_new": true}, {"model_id": "new_h2_NORMAL_XGBoost_N20_t7", "algo": "XGBoost", "regime": "NORMAL", "horizon": 2, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "INTC_ret_5d", "DAX_Germany_zscore_60d", "SPY_zscore_60d", "MO_AltriaMG_ret_1d", "HangSeng_HK_ret_1d", "Nikkei_Japan_vol_20d", "CPB_CampbellSoup_zscore_60d", "EWQ_France_zscore_60d", "3M_vol_20d", "TXN_vol_20d", "PLD_Prologis_ret_5d", "EWA_Australia_ret_1d", "TM_Telephone_ret_1d", "US3M_Rate_vol_20d", "PPL_PPL_ret_1d", "AMD_ret_1d", "heston_var_ev_h3", "PCAR_PaccarInc_ret_5d"], "is_new": true}, {"model_id": "new_h2_NORMAL_XGBoost_N25_t0", "algo": "XGBoost", "regime": "NORMAL", "horizon": 2, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "PPL_PPL_ret_1d", "PCAR_PaccarInc_ret_5d", "Core_CPI_zscore_60d", "Core_PCE_zscore_60d", "SJM_JM_Smucker_ret_5d", "HD_zscore_60d", "DHR_ret_1d", "INTC_ret_5d", "DAX_Germany_zscore_60d", "SBUX_ret_5d", "SLB_Schlumberger_ret_1d", "MS_MorganStanley_ret_1d", "EWC_Canada_zscore_60d", "3M_ret_5d", "XLB_Materials_zscore_60d", "BLK_BlackRock_zscore_60d", "AMD_ret_1d", "PAYX_Paychex_ret_20d", "CI_Cigna_vol_20d", "DIS_vol_20d", "Brent_Oil_FRED_ret_20d", "AXP_Amex_vol_20d", "DHR_vol_20d"], "is_new": true}, {"model_id": "new_h2_NORMAL_XGBoost_N25_t1", "algo": "XGBoost", "regime": "NORMAL", "horizon": 2, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "HUM_Humana_ret_5d", "HD_zscore_60d", "SBUX_zscore_60d", "EWL_Switzerland_vol_20d", "BA_ret_1d", "LMT_LockheedMartin_ret_1d", "SBUX_ret_5d", "PAYX_Paychex_vol_20d", "PPL_PPL_ret_1d", "CPB_CampbellSoup_ret_20d", "XLY_Disc_vol_20d", "Nikkei_Japan_vol_20d", "Core_PCE_zscore_60d", "US6M_Rate_ret_20d", "US7Y_Rate_ret_20d", "QQQ_vol_20d", "CPB_CampbellSoup_zscore_60d", "MSTR_Bitcoin3_ret_1d", "LUV_SouthwestAir_ret_5d", "ORCL_vol_20d", "hmm_p_stress", "M_Macys_vol_20d", "Michigan_Sentiment_ret_20d"], "is_new": true}, {"model_id": "new_h2_NORMAL_XGBoost_N25_t2", "algo": "XGBoost", "regime": "NORMAL", "horizon": 2, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "XLF_Fin_vol_20d", "XLK_Tech_zscore_60d", "MSTR_Bitcoin3_ret_1d", "TED_Spread_vol_20d", "EWQ_France_ret_20d", "EWA_Australia_zscore_60d", "HD_ret_5d", "GD_GeneralDynamics_zscore_60d", "AMGN_Amgen_ret_1d", "heston_var_ev_h3", "XLV_Health_zscore_60d", "LUV_SouthwestAir_ret_5d", "DOW_Price_zscore_60d", "EMR_Emerson_ret_20d", "HD_ret_20d", "DHR_ret_1d", "MSTR_Bitcoin3_ret_20d", "XOM_ret_20d", "EWC_Canada_zscore_60d", "PFE_ret_1d", "EWG_Germany_ret_20d", "EQR_Equity_ret_1d", "DAX_Germany_zscore_60d"], "is_new": true}, {"model_id": "new_h2_NORMAL_XGBoost_N25_t3", "algo": "XGBoost", "regime": "NORMAL", "horizon": 2, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "XOM_ret_1d", "AMD_ret_5d", "PFE_ret_1d", "AMT_AmericanTower_ret_1d", "LOW_Lowes_ret_5d", "HD_ret_20d", "Retail_Sales_zscore_60d", "EWS_Singapore_ret_5d", "NVDA_vol_20d", "AXP_Amex_vol_20d", "Core_CPI_zscore_60d", "IYR_US_REIT2_zscore_60d", "SCHW_Schwab_ret_5d", "VOD_Vodafone_zscore_60d", "gjr_condvar_h1", "CI_Cigna_vol_20d", "IYM_BasicMaterials_ret_20d", "ASX_Australia_vol_20d", "CPB_CampbellSoup_ret_20d", "AXP_Amex_ret_20d", "BTI_BritishAmerican_ret_5d", "MS_MorganStanley_zscore_60d", "DOW_Price_zscore_60d"], "is_new": true}, {"model_id": "new_h2_NORMAL_XGBoost_N25_t4", "algo": "XGBoost", "regime": "NORMAL", "horizon": 2, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "DE_Deere_vol_20d", "M_Macys_vol_20d", "LMT_LockheedMartin_ret_1d", "US30Y_Rate_ret_20d", "US3Y_Rate_ret_5d", "EWH_HongKong_ret_5d", "EXC_Exelon_zscore_60d", "PLD_Prologis_ret_5d", "Industrial_Production_zscore_60d", "WTI_Oil_FRED_zscore_60d", "PCAR_PaccarInc_ret_5d", "US1Y_Rate_ret_5d", "EWG_Germany_vol_20d", "EWL_Switzerland_zscore_60d", "INTC_ret_5d", "HD_ret_5d", "SLB_Schlumberger_ret_1d", "SJM_JM_Smucker_ret_5d", "XOM_ret_1d", "LOW_Lowes_ret_20d", "SBUX_zscore_60d", "CLX_Clorox_vol_20d", "ES_Evergy_ret_1d"], "is_new": true}, {"model_id": "new_h2_NORMAL_XGBoost_N25_t5", "algo": "XGBoost", "regime": "NORMAL", "horizon": 2, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "US3M_Rate_zscore_60d", "CMCSA_ret_1d", "EWY_Korea_zscore_60d", "MSTR_Bitcoin3_ret_5d", "EWS_Singapore_ret_5d", "SLB_Schlumberger_ret_1d", "NFCI_ret_5d", "heston_ev_h3", "MS_MorganStanley_ret_1d", "HD_ret_20d", "EWQ_France_zscore_60d", "EFFR_ret_1d", "AMT_AmericanTower_ret_1d", "EWH_HongKong_ret_5d", "vix_acceleration_1d", "SCHW_Schwab_ret_5d", "DHR_ret_1d", "NEE_NextEra_ret_20d", "NWL_Newell_ret_20d", "LUV_SouthwestAir_ret_5d", "DIS_vol_20d", "SPY_zscore_60d", "DHR_vol_20d"], "is_new": true}, {"model_id": "new_h2_NORMAL_XGBoost_N25_t6", "algo": "XGBoost", "regime": "NORMAL", "horizon": 2, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "VRP_ma5", "ENB_EnbridgeInc_ret_1d", "SJM_JM_Smucker_ret_1d", "DAX_Germany_zscore_60d", "HD_zscore_60d", "3M_ret_5d", "AVB_AvalonBay_zscore_60d", "ASX_Australia_ret_5d", "DHR_vol_20d", "AMT_AmericanTower_ret_1d", "TXN_vol_20d", "MS_MorganStanley_ret_5d", "AXP_Amex_vol_20d", "CLX_Clorox_vol_20d", "HangSeng_HK_ret_5d", "MSTR_Bitcoin3_ret_1d", "AMGN_Amgen_ret_1d", "INTC_ret_5d", "XLY_Disc_vol_20d", "VOD_Vodafone_zscore_60d", "AMZN_ret_5d", "US30Y_Rate_ret_20d", "Industrial_Production_zscore_60d"], "is_new": true}, {"model_id": "new_h2_NORMAL_XGBoost_N25_t7", "algo": "XGBoost", "regime": "NORMAL", "horizon": 2, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "Core_CPI_zscore_60d", "M_Macys_vol_20d", "hmm_p_stress", "EWG_Germany_ret_20d", "BDX_Becton_Dickinson_ret_20d", "heston_var_ev_h7", "EWC_Canada_zscore_60d", "EWQ_France_ret_20d", "INTC_ret_5d", "NVDA_vol_20d", "EWL_Switzerland_vol_20d", "MSTR_Bitcoin3_ret_20d", "WTI_Oil_FRED_zscore_60d", "spx_momentum_3d", "AMD_ret_5d", "CLX_Clorox_vol_20d", "spx_abs_ret_max_5d", "XLF_Fin_vol_20d", "ASX_Australia_vol_20d", "VOD_Vodafone_zscore_60d", "BA_ret_1d", "TED_Spread_vol_20d", "heston_var_ev_h5"], "is_new": true}, {"model_id": "new_h2_NORMAL_XGBoost_N30_t0", "algo": "XGBoost", "regime": "NORMAL", "horizon": 2, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "IBEX_Spain_ret_20d", "TGT_Target_zscore_60d", "AXP_Amex_ret_20d", "EOG_EOGResources_vol_20d", "TED_Spread_vol_20d", "heston_var_ev_h5", "EWM_Malaysia_zscore_60d", "IYR_US_REIT2_zscore_60d", "HD_ret_1d", "BTI_BritishAmerican_ret_20d", "hmm_p_stress", "TXN_vol_20d", "WTI_Oil_FRED_zscore_60d", "EQR_Equity_ret_1d", "AXP_Amex_vol_20d", "EWM_Malaysia_vol_20d", "XLV_Health_zscore_60d", "FedFunds_zscore_60d", "EWL_Switzerland_zscore_60d", "PAYX_Paychex_ret_20d", "NOC_Northrop_ret_20d", "NWL_Newell_ret_20d", "CLX_Clorox_vol_20d", "EWS_Singapore_ret_5d", "VRP_ma5", "MS_MorganStanley_ret_5d", "EOG_EOGResources_ret_5d", "vix_acceleration_1d"], "is_new": true}, {"model_id": "new_h2_NORMAL_XGBoost_N30_t1", "algo": "XGBoost", "regime": "NORMAL", "horizon": 2, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "DHR_ret_1d", "heston_ev_h3", "Nikkei_Japan_vol_20d", "T10Y2Y_Spread_ret_5d", "BTI_BritishAmerican_ret_5d", "LMT_LockheedMartin_ret_1d", "US3Y_Rate_ret_5d", "HangSeng_HK_ret_5d", "PFE_ret_1d", "EXC_Exelon_ret_1d", "heston_var_ev_h3", "3M_ret_5d", "LLY_zscore_60d", "EWL_Switzerland_zscore_60d", "Brent_Oil_FRED_ret_20d", "CLX_Clorox_vol_20d", "ASX_Australia_vol_20d", "hmm_p_stress", "T_ret_1d", "EWJ_Japan_vol_20d", "EWS_Singapore_ret_5d", "AMD_ret_1d", "Nikkei_Japan_zscore_60d", "SJM_JM_Smucker_ret_1d", "EWG_Germany_ret_20d", "PAYX_Paychex_vol_20d", "SJM_JM_Smucker_ret_5d", "HD_zscore_60d"], "is_new": true}, {"model_id": "new_h2_NORMAL_XGBoost_N30_t2", "algo": "XGBoost", "regime": "NORMAL", "horizon": 2, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "DHR_ret_1d", "vix_acceleration_1d", "NWL_Newell_ret_20d", "XLF_Fin_vol_20d", "EWM_Malaysia_zscore_60d", "QQQ_vol_20d", "NEE_NextEra_ret_20d", "LOW_Lowes_ret_20d", "AORD_AUS_zscore_60d", "CPB_CampbellSoup_vol_20d", "Michigan_Sentiment_ret_20d", "SBUX_vol_20d", "Nikkei_Japan_vol_20d", "Retail_Sales_zscore_60d", "CPB_CampbellSoup_ret_5d", "SLB_Schlumberger_ret_5d", "SCHW_Schwab_ret_5d", "XLY_Disc_vol_20d", "MS_MorganStanley_zscore_60d", "DAX_Germany_zscore_60d", "TED_Spread_zscore_60d", "TM_Telephone_vol_20d", "EWL_Switzerland_vol_20d", "heston_var_ev_h7", "EWQ_France_ret_20d", "Nikkei_Japan_zscore_60d", "TXN_vol_20d", "TED_Spread_vol_20d"], "is_new": true}, {"model_id": "new_h2_NORMAL_XGBoost_N30_t3", "algo": "XGBoost", "regime": "NORMAL", "horizon": 2, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "spx_abs_ret_max_5d", "MS_MorganStanley_zscore_60d", "heston_var_ev_h3", "EMR_Emerson_ret_20d", "PFE_ret_1d", "M_Macys_vol_20d", "EOG_EOGResources_vol_20d", "VRP_ma5", "EFFR_vol_20d", "ES_Evergy_ret_1d", "NWL_Newell_ret_20d", "Nikkei_Japan_zscore_60d", "ORCL_vol_20d", "PAYX_Paychex_ret_20d", "SO_SouthernCo_ret_5d", "XLV_Health_zscore_60d", "AMD_ret_1d", "US3M_Rate_zscore_60d", "EWA_Australia_zscore_60d", "PAYX_Paychex_vol_20d", "vix_acceleration_1d", "CI_Cigna_vol_20d", "heston_var_ev_h5", "IWM_SmallCap_vol_20d", "EOG_EOGResources_ret_5d", "heston_ev_h3", "T_ret_1d", "TM_Telephone_ret_1d"], "is_new": true}, {"model_id": "new_h2_NORMAL_XGBoost_N30_t4", "algo": "XGBoost", "regime": "NORMAL", "horizon": 2, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "US6M_Rate_ret_20d", "US7Y_Rate_ret_20d", "EWQ_France_zscore_60d", "INTC_ret_5d", "US3M_Rate_vol_20d", "TM_Telephone_ret_1d", "CPB_CampbellSoup_zscore_60d", "DE_Deere_vol_20d", "ITT_ITTInc_ret_5d", "JNJ_ret_1d", "XOM_ret_1d", "Retail_Sales_zscore_60d", "XOM_ret_20d", "PLD_Prologis_ret_5d", "DOW_Price_zscore_60d", "ORCL_vol_20d", "heston_var_ev_h3", "XLB_Materials_zscore_60d", "SCHW_Schwab_ret_5d", "MSTR_Bitcoin3_ret_20d", "HUM_Humana_ret_5d", "AMZN_ret_5d", "T_ret_1d", "EWA_Australia_ret_1d", "US1Y_Rate_ret_5d", "MO_AltriaMG_ret_1d", "TGT_Target_zscore_60d", "VOD_Vodafone_zscore_60d"], "is_new": true}, {"model_id": "new_h2_NORMAL_XGBoost_N30_t5", "algo": "XGBoost", "regime": "NORMAL", "horizon": 2, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "DAX_Germany_vol_20d", "ORCL_zscore_60d", "CPB_CampbellSoup_zscore_60d", "MS_MorganStanley_ret_1d", "IYM_BasicMaterials_ret_20d", "SBUX_ret_5d", "US1Y_Rate_ret_20d", "US7Y_Rate_ret_20d", "EWA_Australia_ret_1d", "US3Y_Rate_ret_5d", "BTI_BritishAmerican_ret_5d", "DHR_ret_1d", "VRP_ma5", "LMT_LockheedMartin_vol_20d", "MS_MorganStanley_zscore_60d", "LMT_LockheedMartin_ret_1d", "CMCSA_ret_1d", "TGT_Target_zscore_60d", "XLB_Materials_zscore_60d", "vix_acceleration_1d", "CCI_CrownCastle_vol_20d", "Core_CPI_zscore_60d", "HangSeng_HK_vol_20d", "TED_Spread_vol_20d", "Nikkei_Japan_vol_20d", "AVB_AvalonBay_zscore_60d", "BDX_Becton_Dickinson_ret_20d", "LOW_Lowes_ret_5d"], "is_new": true}, {"model_id": "new_h2_NORMAL_XGBoost_N30_t6", "algo": "XGBoost", "regime": "NORMAL", "horizon": 2, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "GE_ret_1d", "VVIX_ret_20d", "heston_var_ev_h3", "EQR_Equity_ret_1d", "T10Y2Y_Spread_ret_5d", "PAYX_Paychex_vol_20d", "US30Y_Rate_ret_20d", "SBUX_zscore_60d", "LMT_LockheedMartin_ret_1d", "PG_ret_20d", "AXP_Amex_vol_20d", "XOM_ret_20d", "EWM_Malaysia_zscore_60d", "AMGN_Amgen_ret_1d", "DHR_vol_20d", "VOD_Vodafone_zscore_60d", "EWC_Canada_zscore_60d", "XLY_Disc_vol_20d", "DAX_Germany_vol_20d", "CPB_CampbellSoup_vol_20d", "DOW_Price_zscore_60d", "JNJ_ret_1d", "AMD_ret_5d", "BTI_BritishAmerican_ret_20d", "SLB_Schlumberger_ret_1d", "MS_MorganStanley_ret_5d", "ASX_Australia_vol_20d", "SCHW_Schwab_ret_5d"], "is_new": true}, {"model_id": "new_h2_NORMAL_XGBoost_N30_t7", "algo": "XGBoost", "regime": "NORMAL", "horizon": 2, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "spx_vol_5d", "heston_var_ev_h5", "WTI_Oil_FRED_zscore_60d", "LLY_zscore_60d", "DOW_Price_zscore_60d", "EQR_Equity_ret_1d", "EMR_Emerson_ret_20d", "EWM_Malaysia_vol_20d", "EWA_Australia_ret_1d", "NEE_NextEra_ret_20d", "XLB_Materials_zscore_60d", "EWH_HongKong_ret_5d", "Core_PCE_zscore_60d", "US30Y_Rate_ret_20d", "AMGN_Amgen_ret_1d", "MSTR_Bitcoin3_ret_5d", "EOG_EOGResources_ret_5d", "AVB_AvalonBay_zscore_60d", "AMD_ret_5d", "SJM_JM_Smucker_ret_1d", "EWC_Canada_zscore_60d", "AMT_AmericanTower_ret_1d", "DE_Deere_vol_20d", "EXC_Exelon_zscore_60d", "EOG_EOGResources_vol_20d", "XLK_Tech_zscore_60d", "NWL_Newell_ret_20d", "DHR_ret_1d"], "is_new": true}, {"model_id": "new_h2_NORMAL_LightGBM_N5_t0", "algo": "LightGBM", "regime": "NORMAL", "horizon": 2, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "PPL_PPL_ret_1d", "heston_ev_h3", "HD_ret_5d"], "is_new": true}, {"model_id": "new_h2_NORMAL_LightGBM_N5_t1", "algo": "LightGBM", "regime": "NORMAL", "horizon": 2, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "heston_var_ev_h5", "GILD_Gilead_ret_20d", "HangSeng_HK_vol_20d"], "is_new": true}, {"model_id": "new_h2_NORMAL_LightGBM_N5_t2", "algo": "LightGBM", "regime": "NORMAL", "horizon": 2, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EXC_Exelon_ret_1d", "AXP_Amex_vol_20d", "MSTR_Bitcoin3_ret_20d"], "is_new": true}, {"model_id": "new_h2_NORMAL_LightGBM_N5_t3", "algo": "LightGBM", "regime": "NORMAL", "horizon": 2, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "heston_var_ev_h7", "HangSeng_HK_ret_5d", "EWY_Korea_zscore_60d"], "is_new": true}, {"model_id": "new_h2_NORMAL_LightGBM_N5_t4", "algo": "LightGBM", "regime": "NORMAL", "horizon": 2, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "BTI_BritishAmerican_ret_20d", "HD_ret_1d", "MSTR_Bitcoin3_ret_20d"], "is_new": true}, {"model_id": "new_h2_NORMAL_LightGBM_N5_t5", "algo": "LightGBM", "regime": "NORMAL", "horizon": 2, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "SO_SouthernCo_ret_5d", "EWC_Canada_zscore_60d", "vix_acceleration_1d"], "is_new": true}, {"model_id": "new_h2_NORMAL_LightGBM_N5_t6", "algo": "LightGBM", "regime": "NORMAL", "horizon": 2, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "US30Y_Rate_ret_20d", "EWM_Malaysia_vol_20d", "spx_vol_5d"], "is_new": true}, {"model_id": "new_h2_NORMAL_LightGBM_N5_t7", "algo": "LightGBM", "regime": "NORMAL", "horizon": 2, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "SCHW_Schwab_ret_5d", "EWL_Switzerland_vol_20d", "ORCL_vol_20d"], "is_new": true}, {"model_id": "new_h2_NORMAL_LightGBM_N8_t0", "algo": "LightGBM", "regime": "NORMAL", "horizon": 2, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EFFR_ret_1d", "AORD_AUS_zscore_60d", "spx_vol_5d", "PFE_ret_1d", "QQQ_vol_20d", "LUV_SouthwestAir_ret_5d"], "is_new": true}, {"model_id": "new_h2_NORMAL_LightGBM_N8_t1", "algo": "LightGBM", "regime": "NORMAL", "horizon": 2, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWS_Singapore_ret_5d", "EFFR_vol_20d", "hmm_p_stress", "NEE_NextEra_ret_20d", "XLK_Tech_zscore_60d", "US30Y_Rate_ret_20d"], "is_new": true}, {"model_id": "new_h2_NORMAL_LightGBM_N8_t2", "algo": "LightGBM", "regime": "NORMAL", "horizon": 2, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "Nikkei_Japan_zscore_60d", "VVIX_ret_20d", "Industrial_Production_zscore_60d", "TM_Telephone_ret_1d", "ASX_Australia_ret_5d", "LOW_Lowes_ret_20d"], "is_new": true}, {"model_id": "new_h2_NORMAL_LightGBM_N8_t3", "algo": "LightGBM", "regime": "NORMAL", "horizon": 2, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "LOW_Lowes_ret_5d", "ASX_Australia_ret_5d", "VVIX_ret_20d", "EOG_EOGResources_vol_20d", "TGT_Target_zscore_60d", "GD_GeneralDynamics_zscore_60d"], "is_new": true}, {"model_id": "new_h2_NORMAL_LightGBM_N8_t4", "algo": "LightGBM", "regime": "NORMAL", "horizon": 2, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "BA_ret_1d", "PFE_ret_1d", "EWC_Canada_zscore_60d", "HD_ret_1d", "EOG_EOGResources_vol_20d", "XLB_Materials_zscore_60d"], "is_new": true}, {"model_id": "new_h2_NORMAL_LightGBM_N8_t5", "algo": "LightGBM", "regime": "NORMAL", "horizon": 2, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWM_Malaysia_vol_20d", "SBUX_zscore_60d", "DHR_vol_20d", "VRP_ma5", "INTC_ret_1d", "GE_ret_1d"], "is_new": true}, {"model_id": "new_h2_NORMAL_LightGBM_N8_t6", "algo": "LightGBM", "regime": "NORMAL", "horizon": 2, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "US30Y_Rate_ret_20d", "AORD_AUS_zscore_60d", "BTI_BritishAmerican_ret_20d", "NVDA_vol_20d", "DHR_ret_1d", "IYR_US_REIT2_zscore_60d"], "is_new": true}, {"model_id": "new_h2_NORMAL_LightGBM_N8_t7", "algo": "LightGBM", "regime": "NORMAL", "horizon": 2, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "ASX_Australia_vol_20d", "DE_Deere_ret_5d", "heston_ev_h3", "3M_vol_20d", "HangSeng_HK_ret_1d", "JNJ_ret_1d"], "is_new": true}, {"model_id": "new_h2_NORMAL_LightGBM_N10_t0", "algo": "LightGBM", "regime": "NORMAL", "horizon": 2, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWH_HongKong_ret_5d", "JNJ_ret_1d", "EWM_Malaysia_vol_20d", "BTI_BritishAmerican_ret_20d", "DHR_ret_1d", "PG_ret_20d", "VVIX_ret_20d", "FedFunds_zscore_60d"], "is_new": true}, {"model_id": "new_h2_NORMAL_LightGBM_N10_t1", "algo": "LightGBM", "regime": "NORMAL", "horizon": 2, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "XLY_Disc_vol_20d", "IYM_BasicMaterials_ret_20d", "EWA_Australia_ret_1d", "CCI_CrownCastle_vol_20d", "CPB_CampbellSoup_ret_5d", "CTAS_Cintas_vol_20d", "DHR_vol_20d", "EWQ_France_zscore_60d"], "is_new": true}, {"model_id": "new_h2_NORMAL_LightGBM_N10_t2", "algo": "LightGBM", "regime": "NORMAL", "horizon": 2, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "ENB_EnbridgeInc_ret_1d", "M_Macys_vol_20d", "LOW_Lowes_ret_20d", "ES_Evergy_ret_1d", "PFE_ret_1d", "FedFunds_zscore_60d", "DOW_Price_zscore_60d", "US1Y_Rate_ret_20d"], "is_new": true}, {"model_id": "new_h2_NORMAL_LightGBM_N10_t3", "algo": "LightGBM", "regime": "NORMAL", "horizon": 2, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "CTAS_Cintas_vol_20d", "BA_ret_1d", "TED_Spread_vol_20d", "LUV_SouthwestAir_ret_5d", "SCHW_Schwab_ret_5d", "EWM_Malaysia_ret_1d", "TED_Spread_zscore_60d", "EWA_Australia_zscore_60d"], "is_new": true}, {"model_id": "new_h2_NORMAL_LightGBM_N10_t4", "algo": "LightGBM", "regime": "NORMAL", "horizon": 2, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "JNJ_ret_1d", "CCI_CrownCastle_vol_20d", "EXC_Exelon_zscore_60d", "gjr_condvar_h1", "Nikkei_Japan_zscore_60d", "AVB_AvalonBay_zscore_60d", "XLB_Materials_zscore_60d", "XOM_ret_20d"], "is_new": true}, {"model_id": "new_h2_NORMAL_LightGBM_N10_t5", "algo": "LightGBM", "regime": "NORMAL", "horizon": 2, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "CCI_CrownCastle_vol_20d", "Retail_Sales_zscore_60d", "vix_acceleration_1d", "HangSeng_HK_ret_5d", "LLY_zscore_60d", "LOW_Lowes_ret_20d", "INTC_ret_1d", "EWL_Switzerland_zscore_60d"], "is_new": true}, {"model_id": "new_h2_NORMAL_LightGBM_N10_t6", "algo": "LightGBM", "regime": "NORMAL", "horizon": 2, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "DE_Deere_vol_20d", "QQQ_vol_20d", "ES_Evergy_ret_1d", "TXN_vol_20d", "CPB_CampbellSoup_zscore_60d", "NOC_Northrop_ret_20d", "PCAR_PaccarInc_ret_5d", "EWG_Germany_ret_20d"], "is_new": true}, {"model_id": "new_h2_NORMAL_LightGBM_N10_t7", "algo": "LightGBM", "regime": "NORMAL", "horizon": 2, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "CPB_CampbellSoup_ret_20d", "DHR_ret_1d", "CLX_Clorox_vol_20d", "vix_acceleration_1d", "INTC_ret_5d", "QQQ_vol_20d", "MRK_Merck_zscore_60d", "NWL_Newell_ret_20d"], "is_new": true}, {"model_id": "new_h2_NORMAL_LightGBM_N12_t0", "algo": "LightGBM", "regime": "NORMAL", "horizon": 2, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "CPB_CampbellSoup_ret_5d", "EOG_EOGResources_vol_20d", "NWL_Newell_ret_20d", "AORD_AUS_zscore_60d", "SO_SouthernCo_ret_5d", "EXC_Exelon_ret_1d", "TED_Spread_vol_20d", "Core_PCE_zscore_60d", "SLB_Schlumberger_ret_5d", "spx_abs_ret_max_5d"], "is_new": true}, {"model_id": "new_h2_NORMAL_LightGBM_N12_t1", "algo": "LightGBM", "regime": "NORMAL", "horizon": 2, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "VOD_Vodafone_zscore_60d", "XOM_ret_1d", "BLK_BlackRock_zscore_60d", "NWL_Newell_ret_20d", "US3Y_Rate_ret_5d", "EWM_Malaysia_zscore_60d", "CI_Cigna_vol_20d", "XLB_Materials_zscore_60d", "Nikkei_Japan_zscore_60d", "EWQ_France_zscore_60d"], "is_new": true}, {"model_id": "new_h2_NORMAL_LightGBM_N12_t2", "algo": "LightGBM", "regime": "NORMAL", "horizon": 2, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "ORCL_vol_20d", "SCHW_Schwab_ret_5d", "BLK_BlackRock_zscore_60d", "IBEX_Spain_ret_20d", "IYR_US_REIT2_zscore_60d", "ASX_Australia_ret_5d", "IYM_BasicMaterials_ret_20d", "XLV_Health_zscore_60d", "VRP_ma5", "TM_Telephone_vol_20d"], "is_new": true}, {"model_id": "new_h2_NORMAL_LightGBM_N12_t3", "algo": "LightGBM", "regime": "NORMAL", "horizon": 2, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWM_Malaysia_vol_20d", "CPB_CampbellSoup_ret_5d", "EOG_EOGResources_vol_20d", "EWL_Switzerland_vol_20d", "HD_ret_1d", "TED_Spread_zscore_60d", "US1Y_Rate_ret_5d", "ASX_Australia_vol_20d", "HangSeng_HK_vol_20d", "SJM_JM_Smucker_ret_5d"], "is_new": true}, {"model_id": "new_h2_NORMAL_LightGBM_N12_t4", "algo": "LightGBM", "regime": "NORMAL", "horizon": 2, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "BDX_Becton_Dickinson_ret_20d", "DIS_vol_20d", "MO_AltriaMG_ret_1d", "XLF_Fin_vol_20d", "EWS_Singapore_ret_5d", "M_Macys_vol_20d", "HangSeng_HK_ret_1d", "LUV_SouthwestAir_ret_5d", "AXP_Amex_ret_20d", "US1Y_Rate_ret_20d"], "is_new": true}, {"model_id": "new_h2_NORMAL_LightGBM_N12_t5", "algo": "LightGBM", "regime": "NORMAL", "horizon": 2, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EXC_Exelon_ret_1d", "DE_Deere_ret_5d", "PPL_PPL_ret_1d", "LOW_Lowes_ret_5d", "MO_AltriaMG_ret_1d", "PAYX_Paychex_ret_20d", "MSTR_Bitcoin3_ret_1d", "PLD_Prologis_ret_5d", "ES_Evergy_ret_1d", "XLF_Fin_vol_20d"], "is_new": true}, {"model_id": "new_h2_NORMAL_LightGBM_N12_t6", "algo": "LightGBM", "regime": "NORMAL", "horizon": 2, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "HD_zscore_60d", "VRP_ma5", "AMZN_ret_5d", "HD_ret_5d", "XOM_ret_20d", "EXC_Exelon_zscore_60d", "EWA_Australia_zscore_60d", "CI_Cigna_vol_20d", "T10Y2Y_Spread_ret_5d", "EWG_Germany_ret_20d"], "is_new": true}, {"model_id": "new_h2_NORMAL_LightGBM_N12_t7", "algo": "LightGBM", "regime": "NORMAL", "horizon": 2, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "XLV_Health_zscore_60d", "MRK_Merck_zscore_60d", "EWM_Malaysia_ret_1d", "CI_Cigna_vol_20d", "ASX_Australia_vol_20d", "spx_abs_ret_max_5d", "EFFR_vol_20d", "vix_acceleration_1d", "US7Y_Rate_ret_20d", "SBUX_ret_5d"], "is_new": true}, {"model_id": "new_h2_NORMAL_LightGBM_N15_t0", "algo": "LightGBM", "regime": "NORMAL", "horizon": 2, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "AMGN_Amgen_ret_1d", "NEE_NextEra_ret_20d", "DIS_vol_20d", "US7Y_Rate_ret_20d", "Core_CPI_zscore_60d", "PLD_Prologis_ret_5d", "XLB_Materials_zscore_60d", "DAX_Germany_zscore_60d", "MS_MorganStanley_ret_1d", "Industrial_Production_zscore_60d", "BTI_BritishAmerican_ret_20d", "PAYX_Paychex_zscore_60d", "EWJ_Japan_vol_20d"], "is_new": true}, {"model_id": "new_h2_NORMAL_LightGBM_N15_t1", "algo": "LightGBM", "regime": "NORMAL", "horizon": 2, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "XLV_Health_zscore_60d", "CI_Cigna_vol_20d", "BA_ret_1d", "SJM_JM_Smucker_ret_5d", "SBUX_zscore_60d", "EWM_Malaysia_vol_20d", "XLY_Disc_vol_20d", "EFFR_ret_1d", "EFFR_vol_20d", "CPB_CampbellSoup_ret_5d", "VRP_ma5", "T_ret_1d", "EWY_Korea_ret_20d"], "is_new": true}, {"model_id": "new_h2_NORMAL_LightGBM_N15_t2", "algo": "LightGBM", "regime": "NORMAL", "horizon": 2, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWL_Switzerland_vol_20d", "PFE_ret_1d", "ORCL_vol_20d", "SBUX_ret_5d", "Core_CPI_zscore_60d", "AMZN_ret_5d", "LOW_Lowes_ret_20d", "DHR_ret_1d", "US3M_Rate_vol_20d", "SO_SouthernCo_ret_5d", "PAYX_Paychex_zscore_60d", "SBUX_vol_20d", "3M_ret_5d"], "is_new": true}, {"model_id": "new_h2_NORMAL_LightGBM_N15_t3", "algo": "LightGBM", "regime": "NORMAL", "horizon": 2, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "MRK_Merck_zscore_60d", "EWC_Canada_zscore_60d", "BDX_Becton_Dickinson_ret_20d", "EXC_Exelon_ret_1d", "GILD_Gilead_ret_20d", "INTC_ret_5d", "BA_ret_1d", "AMD_ret_1d", "EQIX_Equinix_ret_5d", "3M_vol_20d", "DE_Deere_ret_5d", "heston_var_ev_h7", "MSTR_Bitcoin3_ret_1d"], "is_new": true}, {"model_id": "new_h2_NORMAL_LightGBM_N15_t4", "algo": "LightGBM", "regime": "NORMAL", "horizon": 2, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWJ_Japan_vol_20d", "IBEX_Spain_ret_20d", "heston_var_ev_h3", "LLY_zscore_60d", "BDX_Becton_Dickinson_ret_20d", "EOG_EOGResources_ret_5d", "VVIX_ret_20d", "INTC_ret_1d", "SJM_JM_Smucker_ret_5d", "M_Macys_vol_20d", "PAYX_Paychex_zscore_60d", "MS_MorganStanley_zscore_60d", "AXP_Amex_vol_20d"], "is_new": true}, {"model_id": "new_h2_NORMAL_LightGBM_N15_t5", "algo": "LightGBM", "regime": "NORMAL", "horizon": 2, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "XLB_Materials_zscore_60d", "EWG_Germany_vol_20d", "GE_ret_1d", "NEE_NextEra_ret_20d", "PFE_ret_1d", "LMT_LockheedMartin_ret_1d", "HangSeng_HK_ret_1d", "XOM_ret_20d", "BTI_BritishAmerican_ret_5d", "TM_Telephone_ret_1d", "DIS_vol_20d", "EWQ_France_zscore_60d", "NWL_Newell_ret_20d"], "is_new": true}, {"model_id": "new_h2_NORMAL_LightGBM_N15_t6", "algo": "LightGBM", "regime": "NORMAL", "horizon": 2, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWS_Singapore_ret_5d", "T_ret_1d", "vix_acceleration_1d", "DE_Deere_ret_5d", "PAYX_Paychex_vol_20d", "EOG_EOGResources_vol_20d", "Brent_Oil_FRED_ret_20d", "AXP_Amex_vol_20d", "INTC_ret_1d", "CPB_CampbellSoup_ret_5d", "HD_ret_1d", "MSTR_Bitcoin3_ret_1d", "BLK_BlackRock_zscore_60d"], "is_new": true}, {"model_id": "new_h2_NORMAL_LightGBM_N15_t7", "algo": "LightGBM", "regime": "NORMAL", "horizon": 2, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "3M_ret_5d", "CPB_CampbellSoup_ret_5d", "EWQ_France_ret_20d", "TM_Telephone_ret_1d", "spx_momentum_3d", "T_ret_1d", "EOG_EOGResources_ret_5d", "LOW_Lowes_ret_5d", "EOG_EOGResources_vol_20d", "ASX_Australia_ret_5d", "EWC_Canada_zscore_60d", "Brent_Oil_FRED_ret_5d", "MS_MorganStanley_ret_1d"], "is_new": true}, {"model_id": "new_h2_NORMAL_LightGBM_N20_t0", "algo": "LightGBM", "regime": "NORMAL", "horizon": 2, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "NOC_Northrop_ret_20d", "AMD_ret_1d", "EWQ_France_ret_20d", "TM_Telephone_ret_1d", "US3M_Rate_vol_20d", "vix_acceleration_1d", "CLX_Clorox_vol_20d", "hmm_p_stress", "Retail_Sales_zscore_60d", "EWS_Singapore_ret_5d", "MSTR_Bitcoin3_ret_20d", "WTI_Oil_FRED_zscore_60d", "AMD_ret_5d", "PLD_Prologis_ret_5d", "SLB_Schlumberger_ret_5d", "VRP_ma5", "HangSeng_HK_vol_20d", "heston_var_ev_h7"], "is_new": true}, {"model_id": "new_h2_NORMAL_LightGBM_N20_t1", "algo": "LightGBM", "regime": "NORMAL", "horizon": 2, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "AXP_Amex_ret_20d", "AVB_AvalonBay_zscore_60d", "AMT_AmericanTower_ret_1d", "LMT_LockheedMartin_vol_20d", "DE_Deere_ret_5d", "CCI_CrownCastle_vol_20d", "EXC_Exelon_ret_1d", "CPB_CampbellSoup_ret_5d", "AXP_Amex_vol_20d", "QQQ_vol_20d", "AMD_ret_1d", "PCAR_PaccarInc_ret_5d", "Core_CPI_zscore_60d", "ORCL_vol_20d", "EWQ_France_zscore_60d", "GD_GeneralDynamics_zscore_60d", "US30Y_Rate_ret_20d", "LUV_SouthwestAir_ret_5d"], "is_new": true}, {"model_id": "new_h2_NORMAL_LightGBM_N20_t2", "algo": "LightGBM", "regime": "NORMAL", "horizon": 2, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWL_Switzerland_vol_20d", "PFE_ret_1d", "3M_vol_20d", "TED_Spread_zscore_60d", "MRK_Merck_zscore_60d", "CCI_CrownCastle_vol_20d", "CTAS_Cintas_vol_20d", "US7Y_Rate_ret_20d", "XOM_ret_1d", "FedFunds_zscore_60d", "heston_var_ev_h5", "HangSeng_HK_ret_5d", "LOW_Lowes_ret_5d", "EXC_Exelon_zscore_60d", "LUV_SouthwestAir_ret_5d", "NVDA_vol_20d", "XLF_Fin_vol_20d", "PPL_PPL_ret_1d"], "is_new": true}, {"model_id": "new_h2_NORMAL_LightGBM_N20_t3", "algo": "LightGBM", "regime": "NORMAL", "horizon": 2, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "MS_MorganStanley_ret_1d", "Core_CPI_zscore_60d", "WTI_Oil_FRED_zscore_60d", "ES_Evergy_ret_1d", "PCAR_PaccarInc_ret_5d", "Nikkei_Japan_vol_20d", "SJM_JM_Smucker_ret_1d", "CCI_CrownCastle_vol_20d", "EWQ_France_ret_20d", "ORCL_zscore_60d", "PAYX_Paychex_vol_20d", "SJM_JM_Smucker_ret_5d", "NOC_Northrop_ret_20d", "DHR_vol_20d", "NWL_Newell_ret_20d", "EWQ_France_zscore_60d", "US7Y_Rate_ret_20d", "LMT_LockheedMartin_vol_20d"], "is_new": true}, {"model_id": "new_h2_NORMAL_LightGBM_N20_t4", "algo": "LightGBM", "regime": "NORMAL", "horizon": 2, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "MSTR_Bitcoin3_ret_20d", "ORCL_zscore_60d", "TXN_vol_20d", "EWL_Switzerland_zscore_60d", "AMZN_ret_5d", "MRK_Merck_zscore_60d", "LUV_SouthwestAir_ret_5d", "SCHW_Schwab_ret_5d", "EWH_HongKong_ret_5d", "PFE_ret_1d", "CPB_CampbellSoup_ret_5d", "EWC_Canada_zscore_60d", "TED_Spread_vol_20d", "Retail_Sales_zscore_60d", "HD_ret_5d", "AXP_Amex_ret_20d", "ENB_EnbridgeInc_ret_1d", "DHR_ret_1d"], "is_new": true}, {"model_id": "new_h2_NORMAL_LightGBM_N20_t5", "algo": "LightGBM", "regime": "NORMAL", "horizon": 2, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "HangSeng_HK_ret_5d", "HangSeng_HK_ret_1d", "MS_MorganStanley_ret_5d", "MSTR_Bitcoin3_ret_20d", "EQIX_Equinix_ret_5d", "SCHW_Schwab_ret_5d", "NWL_Newell_ret_20d", "EWG_Germany_vol_20d", "EWL_Switzerland_vol_20d", "3M_ret_5d", "HUM_Humana_ret_5d", "CTAS_Cintas_vol_20d", "Brent_Oil_FRED_ret_20d", "PFE_ret_1d", "NOC_Northrop_ret_20d", "CPB_CampbellSoup_ret_20d", "EOG_EOGResources_vol_20d", "XOM_ret_20d"], "is_new": true}, {"model_id": "new_h2_NORMAL_LightGBM_N20_t6", "algo": "LightGBM", "regime": "NORMAL", "horizon": 2, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "HUM_Humana_ret_5d", "SLB_Schlumberger_ret_5d", "EWM_Malaysia_vol_20d", "CTAS_Cintas_vol_20d", "WTI_Oil_FRED_zscore_60d", "MS_MorganStanley_zscore_60d", "TGT_Target_zscore_60d", "NVDA_vol_20d", "EWH_HongKong_ret_5d", "NOC_Northrop_ret_20d", "FedFunds_zscore_60d", "Retail_Sales_zscore_60d", "AMZN_ret_5d", "EWY_Korea_zscore_60d", "MS_MorganStanley_ret_1d", "INTC_ret_1d", "MSTR_Bitcoin3_ret_1d", "TM_Telephone_ret_1d"], "is_new": true}, {"model_id": "new_h2_NORMAL_LightGBM_N20_t7", "algo": "LightGBM", "regime": "NORMAL", "horizon": 2, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWL_Switzerland_vol_20d", "MSTR_Bitcoin3_ret_1d", "US1Y_Rate_ret_20d", "US30Y_Rate_ret_20d", "EWM_Malaysia_zscore_60d", "3M_ret_5d", "AORD_AUS_zscore_60d", "HD_ret_1d", "TM_Telephone_vol_20d", "EWL_Switzerland_zscore_60d", "SBUX_vol_20d", "XLB_Materials_zscore_60d", "heston_var_ev_h5", "EWM_Malaysia_vol_20d", "M_Macys_vol_20d", "CCI_CrownCastle_vol_20d", "LLY_zscore_60d", "US5Y_Rate_ret_5d"], "is_new": true}, {"model_id": "new_h2_NORMAL_LightGBM_N25_t0", "algo": "LightGBM", "regime": "NORMAL", "horizon": 2, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "SBUX_ret_5d", "BDX_Becton_Dickinson_ret_20d", "MSTR_Bitcoin3_ret_5d", "IYM_BasicMaterials_ret_20d", "WTI_Oil_FRED_zscore_60d", "EWM_Malaysia_ret_1d", "HD_ret_1d", "BTI_BritishAmerican_ret_20d", "AXP_Amex_ret_20d", "FedFunds_zscore_60d", "AXP_Amex_vol_20d", "MSTR_Bitcoin3_ret_1d", "DIS_vol_20d", "US6M_Rate_ret_20d", "CPB_CampbellSoup_ret_20d", "CCI_CrownCastle_vol_20d", "TXN_vol_20d", "PAYX_Paychex_zscore_60d", "EOG_EOGResources_ret_5d", "DAX_Germany_zscore_60d", "SO_SouthernCo_ret_5d", "EFFR_vol_20d", "heston_var_ev_h7"], "is_new": true}, {"model_id": "new_h2_NORMAL_LightGBM_N25_t1", "algo": "LightGBM", "regime": "NORMAL", "horizon": 2, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "CLX_Clorox_vol_20d", "TM_Telephone_ret_1d", "Core_CPI_zscore_60d", "PAYX_Paychex_vol_20d", "US7Y_Rate_ret_20d", "M_Macys_vol_20d", "BA_ret_1d", "Retail_Sales_zscore_60d", "IYR_US_REIT2_zscore_60d", "spx_momentum_3d", "TED_Spread_vol_20d", "XLK_Tech_zscore_60d", "T10Y2Y_Spread_ret_5d", "EWY_Korea_zscore_60d", "US5Y_Rate_ret_5d", "heston_var_ev_h5", "HD_ret_20d", "US6M_Rate_ret_20d", "heston_ev_h3", "BDX_Becton_Dickinson_ret_20d", "INTC_ret_1d", "EMR_Emerson_ret_20d", "DE_Deere_vol_20d"], "is_new": true}, {"model_id": "new_h2_NORMAL_LightGBM_N25_t2", "algo": "LightGBM", "regime": "NORMAL", "horizon": 2, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "HD_ret_1d", "TXN_vol_20d", "NEE_NextEra_ret_20d", "IYR_US_REIT2_zscore_60d", "NOC_Northrop_ret_20d", "LOW_Lowes_ret_20d", "SPY_zscore_60d", "CPB_CampbellSoup_ret_20d", "US1Y_Rate_ret_20d", "XLY_Disc_vol_20d", "TGT_Target_zscore_60d", "CPB_CampbellSoup_vol_20d", "AMD_ret_1d", "Nikkei_Japan_zscore_60d", "Brent_Oil_FRED_ret_5d", "SCHW_Schwab_ret_5d", "EWJ_Japan_vol_20d", "INTC_ret_1d", "spx_vol_5d", "EWS_Singapore_ret_5d", "heston_var_ev_h5", "WTI_Oil_FRED_zscore_60d", "EWG_Germany_ret_20d"], "is_new": true}, {"model_id": "new_h2_NORMAL_LightGBM_N25_t3", "algo": "LightGBM", "regime": "NORMAL", "horizon": 2, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "LMT_LockheedMartin_vol_20d", "PAYX_Paychex_vol_20d", "EQIX_Equinix_ret_5d", "NOC_Northrop_ret_20d", "CPB_CampbellSoup_vol_20d", "NVDA_vol_20d", "VVIX_ret_20d", "HangSeng_HK_vol_20d", "XLF_Fin_vol_20d", "NFCI_ret_5d", "EWL_Switzerland_vol_20d", "SBUX_vol_20d", "US30Y_Rate_ret_20d", "SLB_Schlumberger_ret_1d", "Brent_Oil_FRED_ret_20d", "CLX_Clorox_vol_20d", "US3Y_Rate_ret_5d", "EWJ_Japan_vol_20d", "TED_Spread_vol_20d", "VRP_ma5", "US6M_Rate_ret_20d", "US7Y_Rate_ret_20d", "BA_ret_1d"], "is_new": true}, {"model_id": "new_h2_NORMAL_LightGBM_N25_t4", "algo": "LightGBM", "regime": "NORMAL", "horizon": 2, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWH_HongKong_ret_5d", "heston_var_ev_h5", "AMD_ret_5d", "SJM_JM_Smucker_ret_1d", "Brent_Oil_FRED_ret_5d", "Nikkei_Japan_vol_20d", "SCHW_Schwab_ret_5d", "LMT_LockheedMartin_ret_1d", "EWA_Australia_ret_1d", "EOG_EOGResources_ret_5d", "Core_CPI_zscore_60d", "MS_MorganStanley_ret_1d", "EWL_Switzerland_zscore_60d", "EWA_Australia_zscore_60d", "NVDA_vol_20d", "Core_PCE_zscore_60d", "EWQ_France_zscore_60d", "spx_abs_ret_max_5d", "PPL_PPL_ret_1d", "TED_Spread_vol_20d", "IYM_BasicMaterials_ret_20d", "GD_GeneralDynamics_zscore_60d", "US6M_Rate_ret_20d"], "is_new": true}, {"model_id": "new_h2_NORMAL_LightGBM_N25_t5", "algo": "LightGBM", "regime": "NORMAL", "horizon": 2, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWS_Singapore_ret_5d", "QQQ_vol_20d", "EMR_Emerson_ret_20d", "WTI_Oil_FRED_zscore_60d", "LOW_Lowes_ret_20d", "HUM_Humana_ret_5d", "MRK_Merck_zscore_60d", "SPY_zscore_60d", "IYR_US_REIT2_zscore_60d", "SJM_JM_Smucker_ret_5d", "EWM_Malaysia_ret_1d", "JNJ_ret_1d", "NVDA_vol_20d", "EWQ_France_ret_20d", "spx_momentum_3d", "VOD_Vodafone_zscore_60d", "T_ret_1d", "DAX_Germany_vol_20d", "EFFR_ret_1d", "SBUX_ret_5d", "HD_ret_1d", "EWL_Switzerland_vol_20d", "XLV_Health_zscore_60d"], "is_new": true}, {"model_id": "new_h2_NORMAL_LightGBM_N25_t6", "algo": "LightGBM", "regime": "NORMAL", "horizon": 2, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "ORCL_zscore_60d", "GE_ret_1d", "BTI_BritishAmerican_ret_5d", "WTI_Oil_FRED_zscore_60d", "ES_Evergy_ret_1d", "ITT_ITTInc_ret_5d", "US1Y_Rate_ret_5d", "spx_vol_5d", "gjr_condvar_h1", "EXC_Exelon_ret_1d", "SCHW_Schwab_ret_5d", "NVDA_vol_20d", "EQIX_Equinix_ret_5d", "CPB_CampbellSoup_zscore_60d", "EQR_Equity_ret_1d", "EWA_Australia_ret_1d", "Michigan_Sentiment_ret_20d", "ASX_Australia_vol_20d", "DAX_Germany_vol_20d", "TM_Telephone_vol_20d", "MS_MorganStanley_zscore_60d", "EWM_Malaysia_zscore_60d", "US3Y_Rate_ret_5d"], "is_new": true}, {"model_id": "new_h2_NORMAL_LightGBM_N25_t7", "algo": "LightGBM", "regime": "NORMAL", "horizon": 2, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "CPB_CampbellSoup_ret_5d", "EWH_HongKong_ret_5d", "US1Y_Rate_ret_5d", "EWY_Korea_zscore_60d", "INTC_ret_5d", "EWM_Malaysia_zscore_60d", "CPB_CampbellSoup_ret_20d", "INTC_ret_1d", "MS_MorganStanley_ret_1d", "EWL_Switzerland_vol_20d", "BA_ret_1d", "XLB_Materials_zscore_60d", "EWS_Singapore_ret_5d", "AMD_ret_1d", "PPL_PPL_ret_1d", "WTI_Oil_FRED_zscore_60d", "M_Macys_vol_20d", "T10Y2Y_Spread_ret_5d", "Industrial_Production_zscore_60d", "AXP_Amex_ret_20d", "VOD_Vodafone_zscore_60d", "Nikkei_Japan_vol_20d", "HangSeng_HK_vol_20d"], "is_new": true}, {"model_id": "new_h2_NORMAL_LightGBM_N30_t0", "algo": "LightGBM", "regime": "NORMAL", "horizon": 2, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "ASX_Australia_ret_5d", "US30Y_Rate_ret_20d", "XLK_Tech_zscore_60d", "XLB_Materials_zscore_60d", "EQIX_Equinix_ret_5d", "CPB_CampbellSoup_zscore_60d", "PFE_ret_1d", "SBUX_ret_5d", "heston_ev_h3", "EWA_Australia_ret_1d", "IYM_BasicMaterials_ret_20d", "CMCSA_ret_1d", "NVDA_vol_20d", "M_Macys_vol_20d", "SLB_Schlumberger_ret_5d", "BTI_BritishAmerican_ret_20d", "BTI_BritishAmerican_ret_5d", "EWG_Germany_vol_20d", "EWL_Switzerland_vol_20d", "PPL_PPL_ret_1d", "EWA_Australia_zscore_60d", "Nikkei_Japan_vol_20d", "PAYX_Paychex_zscore_60d", "AORD_AUS_zscore_60d", "Retail_Sales_zscore_60d", "DE_Deere_vol_20d", "Core_PCE_zscore_60d", "US5Y_Rate_ret_5d"], "is_new": true}, {"model_id": "new_h2_NORMAL_LightGBM_N30_t1", "algo": "LightGBM", "regime": "NORMAL", "horizon": 2, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EQR_Equity_ret_1d", "HangSeng_HK_vol_20d", "DE_Deere_ret_5d", "hmm_p_stress", "EOG_EOGResources_ret_5d", "heston_var_ev_h7", "spx_abs_ret_max_5d", "GE_ret_1d", "CPB_CampbellSoup_ret_20d", "XLV_Health_zscore_60d", "TED_Spread_vol_20d", "EWM_Malaysia_vol_20d", "BLK_BlackRock_zscore_60d", "NFCI_ret_5d", "EWS_Singapore_ret_5d", "ES_Evergy_ret_1d", "EWQ_France_ret_20d", "ORCL_vol_20d", "INTC_ret_1d", "vix_acceleration_1d", "AMD_ret_5d", "heston_var_ev_h3", "TXN_vol_20d", "CPB_CampbellSoup_ret_5d", "IBEX_Spain_ret_20d", "HD_zscore_60d", "GD_GeneralDynamics_zscore_60d", "spx_momentum_3d"], "is_new": true}, {"model_id": "new_h2_NORMAL_LightGBM_N30_t2", "algo": "LightGBM", "regime": "NORMAL", "horizon": 2, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EFFR_vol_20d", "HD_ret_1d", "EWS_Singapore_ret_5d", "MO_AltriaMG_ret_1d", "3M_vol_20d", "WTI_Oil_FRED_zscore_60d", "HangSeng_HK_vol_20d", "ORCL_zscore_60d", "NFCI_ret_5d", "LMT_LockheedMartin_ret_1d", "CMCSA_ret_1d", "Core_PCE_zscore_60d", "SO_SouthernCo_ret_5d", "Nikkei_Japan_vol_20d", "BTI_BritishAmerican_ret_20d", "Industrial_Production_zscore_60d", "NOC_Northrop_ret_20d", "GILD_Gilead_ret_20d", "AMD_ret_1d", "US5Y_Rate_ret_5d", "HD_ret_20d", "CPB_CampbellSoup_vol_20d", "IBEX_Spain_ret_20d", "SCHW_Schwab_ret_5d", "heston_ev_h3", "EWA_Australia_zscore_60d", "TM_Telephone_vol_20d", "XLV_Health_zscore_60d"], "is_new": true}, {"model_id": "new_h2_NORMAL_LightGBM_N30_t3", "algo": "LightGBM", "regime": "NORMAL", "horizon": 2, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "IBEX_Spain_ret_20d", "heston_ev_h3", "DE_Deere_vol_20d", "HangSeng_HK_ret_1d", "IWM_SmallCap_vol_20d", "CMCSA_ret_1d", "vix_acceleration_1d", "XOM_ret_1d", "BDX_Becton_Dickinson_ret_20d", "SLB_Schlumberger_ret_5d", "EQIX_Equinix_ret_5d", "MSTR_Bitcoin3_ret_5d", "Brent_Oil_FRED_ret_5d", "Brent_Oil_FRED_ret_20d", "AMZN_ret_5d", "EWG_Germany_ret_20d", "INTC_ret_5d", "XOM_ret_20d", "XLF_Fin_vol_20d", "GD_GeneralDynamics_zscore_60d", "EQR_Equity_ret_1d", "US5Y_Rate_ret_5d", "EOG_EOGResources_vol_20d", "ASX_Australia_vol_20d", "US7Y_Rate_ret_20d", "EWQ_France_zscore_60d", "Retail_Sales_zscore_60d", "MSTR_Bitcoin3_ret_20d"], "is_new": true}, {"model_id": "new_h2_NORMAL_LightGBM_N30_t4", "algo": "LightGBM", "regime": "NORMAL", "horizon": 2, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "WTI_Oil_FRED_zscore_60d", "EOG_EOGResources_vol_20d", "ENB_EnbridgeInc_ret_1d", "US3M_Rate_zscore_60d", "LLY_zscore_60d", "EWM_Malaysia_zscore_60d", "hmm_p_stress", "TM_Telephone_ret_1d", "MSTR_Bitcoin3_ret_5d", "CMCSA_ret_1d", "spx_momentum_3d", "EWY_Korea_zscore_60d", "heston_ev_h3", "BTI_BritishAmerican_ret_20d", "TM_Telephone_vol_20d", "EWA_Australia_zscore_60d", "XLF_Fin_vol_20d", "AMGN_Amgen_ret_1d", "heston_var_ev_h7", "HangSeng_HK_vol_20d", "DAX_Germany_zscore_60d", "HD_ret_1d", "DOW_Price_zscore_60d", "HUM_Humana_ret_5d", "3M_ret_5d", "MSTR_Bitcoin3_ret_1d", "NWL_Newell_ret_20d", "EQIX_Equinix_ret_5d"], "is_new": true}, {"model_id": "new_h2_NORMAL_LightGBM_N30_t5", "algo": "LightGBM", "regime": "NORMAL", "horizon": 2, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWA_Australia_ret_1d", "EWG_Germany_ret_20d", "HangSeng_HK_ret_1d", "SPY_zscore_60d", "TM_Telephone_ret_1d", "XLB_Materials_zscore_60d", "EWM_Malaysia_vol_20d", "EWQ_France_ret_20d", "EXC_Exelon_ret_1d", "EWH_HongKong_ret_5d", "PPL_PPL_ret_1d", "MO_AltriaMG_ret_1d", "CPB_CampbellSoup_vol_20d", "TED_Spread_vol_20d", "MSTR_Bitcoin3_ret_20d", "ORCL_vol_20d", "heston_var_ev_h3", "ASX_Australia_vol_20d", "SLB_Schlumberger_ret_1d", "XLK_Tech_zscore_60d", "LOW_Lowes_ret_5d", "HD_ret_5d", "EOG_EOGResources_vol_20d", "HD_ret_20d", "EWY_Korea_ret_20d", "US30Y_Rate_ret_20d", "EQIX_Equinix_ret_5d", "MSTR_Bitcoin3_ret_5d"], "is_new": true}, {"model_id": "new_h2_NORMAL_LightGBM_N30_t6", "algo": "LightGBM", "regime": "NORMAL", "horizon": 2, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "AMT_AmericanTower_ret_1d", "PAYX_Paychex_zscore_60d", "GD_GeneralDynamics_zscore_60d", "EWG_Germany_vol_20d", "JNJ_ret_1d", "vix_mean_abs_ret_5d", "US5Y_Rate_ret_5d", "LLY_zscore_60d", "EMR_Emerson_ret_20d", "US30Y_Rate_ret_20d", "T10Y2Y_Spread_ret_5d", "XOM_ret_1d", "MRK_Merck_zscore_60d", "XLF_Fin_vol_20d", "GE_ret_1d", "Retail_Sales_zscore_60d", "AMD_ret_1d", "EWY_Korea_ret_20d", "US3M_Rate_zscore_60d", "Nikkei_Japan_vol_20d", "NWL_Newell_ret_20d", "HD_ret_20d", "IWM_SmallCap_vol_20d", "EWH_HongKong_ret_5d", "ITT_ITTInc_ret_5d", "CPB_CampbellSoup_ret_20d", "PPL_PPL_ret_1d", "Michigan_Sentiment_ret_20d"], "is_new": true}, {"model_id": "new_h2_NORMAL_LightGBM_N30_t7", "algo": "LightGBM", "regime": "NORMAL", "horizon": 2, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "DAX_Germany_vol_20d", "T_ret_1d", "T10Y2Y_Spread_ret_5d", "EWG_Germany_ret_20d", "3M_vol_20d", "ASX_Australia_vol_20d", "VRP_ma5", "VOD_Vodafone_zscore_60d", "JNJ_ret_1d", "GILD_Gilead_ret_20d", "DE_Deere_vol_20d", "vix_acceleration_1d", "SLB_Schlumberger_ret_5d", "PCAR_PaccarInc_ret_5d", "EWJ_Japan_vol_20d", "heston_ev_h3", "HD_ret_20d", "M_Macys_vol_20d", "MSTR_Bitcoin3_ret_1d", "SPY_zscore_60d", "ORCL_vol_20d", "Nikkei_Japan_zscore_60d", "PAYX_Paychex_zscore_60d", "TM_Telephone_ret_1d", "EXC_Exelon_zscore_60d", "XLY_Disc_vol_20d", "AMGN_Amgen_ret_1d", "NEE_NextEra_ret_20d"], "is_new": true}, {"model_id": "new_h2_NORMAL_GradientBoosting_N5_t0", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 2, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "Nikkei_Japan_zscore_60d", "US7Y_Rate_ret_20d", "BA_ret_1d"], "is_new": true}, {"model_id": "new_h2_NORMAL_GradientBoosting_N5_t1", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 2, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "CCI_CrownCastle_vol_20d", "AMZN_ret_5d", "EWL_Switzerland_vol_20d"], "is_new": true}, {"model_id": "new_h2_NORMAL_GradientBoosting_N5_t2", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 2, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "Industrial_Production_zscore_60d", "MS_MorganStanley_ret_5d", "Retail_Sales_zscore_60d"], "is_new": true}, {"model_id": "new_h2_NORMAL_GradientBoosting_N5_t3", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 2, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "3M_ret_5d", "spx_abs_ret_max_5d", "Industrial_Production_zscore_60d"], "is_new": true}, {"model_id": "new_h2_NORMAL_GradientBoosting_N5_t4", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 2, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "MSTR_Bitcoin3_ret_5d", "SBUX_zscore_60d", "spx_vol_5d"], "is_new": true}, {"model_id": "new_h2_NORMAL_GradientBoosting_N5_t5", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 2, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "HD_ret_20d", "EWH_HongKong_ret_5d", "MO_AltriaMG_ret_1d"], "is_new": true}, {"model_id": "new_h2_NORMAL_GradientBoosting_N5_t6", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 2, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "HD_ret_1d", "ITT_ITTInc_ret_5d", "AMGN_Amgen_ret_1d"], "is_new": true}, {"model_id": "new_h2_NORMAL_GradientBoosting_N5_t7", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 2, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "US5Y_Rate_ret_5d", "EWM_Malaysia_vol_20d", "SBUX_vol_20d"], "is_new": true}, {"model_id": "new_h2_NORMAL_GradientBoosting_N8_t0", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 2, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "FedFunds_zscore_60d", "EQIX_Equinix_ret_5d", "US30Y_Rate_ret_20d", "ORCL_vol_20d", "Brent_Oil_FRED_ret_20d", "PPL_PPL_ret_1d"], "is_new": true}, {"model_id": "new_h2_NORMAL_GradientBoosting_N8_t1", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 2, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "DAX_Germany_vol_20d", "HangSeng_HK_ret_1d", "ASX_Australia_ret_5d", "DE_Deere_ret_5d", "IYM_BasicMaterials_ret_20d", "BDX_Becton_Dickinson_ret_20d"], "is_new": true}, {"model_id": "new_h2_NORMAL_GradientBoosting_N8_t2", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 2, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "Nikkei_Japan_vol_20d", "EOG_EOGResources_vol_20d", "MSTR_Bitcoin3_ret_1d", "HangSeng_HK_ret_5d", "SO_SouthernCo_ret_5d", "Brent_Oil_FRED_ret_20d"], "is_new": true}, {"model_id": "new_h2_NORMAL_GradientBoosting_N8_t3", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 2, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "US3Y_Rate_ret_5d", "EWS_Singapore_ret_5d", "NEE_NextEra_ret_20d", "US30Y_Rate_ret_20d", "BA_ret_1d", "vix_mean_abs_ret_5d"], "is_new": true}, {"model_id": "new_h2_NORMAL_GradientBoosting_N8_t4", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 2, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWL_Switzerland_zscore_60d", "AMGN_Amgen_ret_1d", "CI_Cigna_vol_20d", "AMT_AmericanTower_ret_1d", "CCI_CrownCastle_vol_20d", "DHR_vol_20d"], "is_new": true}, {"model_id": "new_h2_NORMAL_GradientBoosting_N8_t5", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 2, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "XLK_Tech_zscore_60d", "hmm_p_stress", "EWH_HongKong_ret_5d", "3M_ret_5d", "HUM_Humana_ret_5d", "heston_var_ev_h5"], "is_new": true}, {"model_id": "new_h2_NORMAL_GradientBoosting_N8_t6", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 2, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "PFE_ret_1d", "LMT_LockheedMartin_ret_1d", "TED_Spread_vol_20d", "MSTR_Bitcoin3_ret_1d", "QQQ_vol_20d", "AXP_Amex_vol_20d"], "is_new": true}, {"model_id": "new_h2_NORMAL_GradientBoosting_N8_t7", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 2, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "DIS_vol_20d", "BA_ret_1d", "EWA_Australia_zscore_60d", "NWL_Newell_ret_20d", "EWL_Switzerland_vol_20d", "EWY_Korea_zscore_60d"], "is_new": true}, {"model_id": "new_h2_NORMAL_GradientBoosting_N10_t0", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 2, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "Michigan_Sentiment_ret_20d", "SLB_Schlumberger_ret_5d", "WTI_Oil_FRED_zscore_60d", "MSTR_Bitcoin3_ret_1d", "EWS_Singapore_ret_5d", "EWM_Malaysia_zscore_60d", "CPB_CampbellSoup_vol_20d", "TM_Telephone_ret_1d"], "is_new": true}, {"model_id": "new_h2_NORMAL_GradientBoosting_N10_t1", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 2, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "MO_AltriaMG_ret_1d", "ORCL_zscore_60d", "NVDA_vol_20d", "CLX_Clorox_vol_20d", "Retail_Sales_zscore_60d", "AXP_Amex_vol_20d", "PAYX_Paychex_vol_20d", "EWL_Switzerland_zscore_60d"], "is_new": true}, {"model_id": "new_h2_NORMAL_GradientBoosting_N10_t2", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 2, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "LOW_Lowes_ret_20d", "PG_ret_20d", "Nikkei_Japan_vol_20d", "EOG_EOGResources_vol_20d", "NFCI_ret_5d", "DE_Deere_vol_20d", "MS_MorganStanley_ret_1d", "EWS_Singapore_ret_5d"], "is_new": true}, {"model_id": "new_h2_NORMAL_GradientBoosting_N10_t3", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 2, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "SLB_Schlumberger_ret_5d", "XLV_Health_zscore_60d", "US1Y_Rate_ret_20d", "XLB_Materials_zscore_60d", "SBUX_zscore_60d", "SCHW_Schwab_ret_5d", "FedFunds_zscore_60d", "MRK_Merck_zscore_60d"], "is_new": true}, {"model_id": "new_h2_NORMAL_GradientBoosting_N10_t4", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 2, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "HD_ret_5d", "SJM_JM_Smucker_ret_5d", "IYR_US_REIT2_zscore_60d", "DHR_ret_1d", "VOD_Vodafone_zscore_60d", "LMT_LockheedMartin_vol_20d", "EWA_Australia_ret_1d", "PAYX_Paychex_vol_20d"], "is_new": true}, {"model_id": "new_h2_NORMAL_GradientBoosting_N10_t5", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 2, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWQ_France_zscore_60d", "QQQ_vol_20d", "MS_MorganStanley_ret_1d", "PLD_Prologis_ret_5d", "NVDA_vol_20d", "US1Y_Rate_ret_5d", "HD_ret_20d", "AMZN_ret_5d"], "is_new": true}, {"model_id": "new_h2_NORMAL_GradientBoosting_N10_t6", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 2, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "NVDA_vol_20d", "PLD_Prologis_ret_5d", "LMT_LockheedMartin_vol_20d", "CCI_CrownCastle_vol_20d", "HangSeng_HK_vol_20d", "VVIX_ret_20d", "PAYX_Paychex_ret_20d", "LMT_LockheedMartin_ret_1d"], "is_new": true}, {"model_id": "new_h2_NORMAL_GradientBoosting_N10_t7", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 2, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "INTC_ret_5d", "PPL_PPL_ret_1d", "EWH_HongKong_ret_5d", "hmm_p_stress", "BLK_BlackRock_zscore_60d", "XOM_ret_1d", "VOD_Vodafone_zscore_60d", "BDX_Becton_Dickinson_ret_20d"], "is_new": true}, {"model_id": "new_h2_NORMAL_GradientBoosting_N12_t0", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 2, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "NEE_NextEra_ret_20d", "MRK_Merck_zscore_60d", "MS_MorganStanley_zscore_60d", "LOW_Lowes_ret_20d", "SLB_Schlumberger_ret_1d", "DOW_Price_zscore_60d", "EWQ_France_zscore_60d", "HD_zscore_60d", "FedFunds_zscore_60d", "JNJ_ret_1d"], "is_new": true}, {"model_id": "new_h2_NORMAL_GradientBoosting_N12_t1", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 2, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWS_Singapore_ret_5d", "JNJ_ret_1d", "EQR_Equity_ret_1d", "EWG_Germany_ret_20d", "DE_Deere_ret_5d", "NFCI_ret_5d", "Nikkei_Japan_zscore_60d", "ASX_Australia_ret_5d", "gjr_condvar_h1", "vix_mean_abs_ret_5d"], "is_new": true}, {"model_id": "new_h2_NORMAL_GradientBoosting_N12_t2", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 2, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "AORD_AUS_zscore_60d", "US3Y_Rate_ret_5d", "CCI_CrownCastle_vol_20d", "BTI_BritishAmerican_ret_5d", "BLK_BlackRock_zscore_60d", "EWS_Singapore_ret_5d", "HangSeng_HK_vol_20d", "US7Y_Rate_ret_20d", "vix_acceleration_1d", "LOW_Lowes_ret_5d"], "is_new": true}, {"model_id": "new_h2_NORMAL_GradientBoosting_N12_t3", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 2, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "INTC_ret_1d", "CCI_CrownCastle_vol_20d", "CLX_Clorox_vol_20d", "T_ret_1d", "CPB_CampbellSoup_vol_20d", "INTC_ret_5d", "SJM_JM_Smucker_ret_5d", "EWL_Switzerland_vol_20d", "PAYX_Paychex_vol_20d", "GILD_Gilead_ret_20d"], "is_new": true}, {"model_id": "new_h2_NORMAL_GradientBoosting_N12_t4", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 2, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "HangSeng_HK_vol_20d", "PPL_PPL_ret_1d", "PAYX_Paychex_vol_20d", "NWL_Newell_ret_20d", "CMCSA_ret_1d", "SLB_Schlumberger_ret_5d", "heston_var_ev_h5", "heston_var_ev_h7", "NEE_NextEra_ret_20d", "SBUX_vol_20d"], "is_new": true}, {"model_id": "new_h2_NORMAL_GradientBoosting_N12_t5", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 2, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "US7Y_Rate_ret_20d", "NFCI_ret_5d", "M_Macys_vol_20d", "vix_mean_abs_ret_5d", "hmm_p_stress", "MSTR_Bitcoin3_ret_5d", "T_ret_1d", "DIS_vol_20d", "XLV_Health_zscore_60d", "AMGN_Amgen_ret_1d"], "is_new": true}, {"model_id": "new_h2_NORMAL_GradientBoosting_N12_t6", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 2, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "Michigan_Sentiment_ret_20d", "JNJ_ret_1d", "WTI_Oil_FRED_zscore_60d", "VOD_Vodafone_zscore_60d", "MRK_Merck_zscore_60d", "AMGN_Amgen_ret_1d", "TED_Spread_vol_20d", "heston_var_ev_h5", "XLY_Disc_vol_20d", "BTI_BritishAmerican_ret_5d"], "is_new": true}, {"model_id": "new_h2_NORMAL_GradientBoosting_N12_t7", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 2, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "3M_ret_5d", "EQR_Equity_ret_1d", "US5Y_Rate_ret_5d", "ES_Evergy_ret_1d", "SJM_JM_Smucker_ret_1d", "NWL_Newell_ret_20d", "HD_ret_5d", "AMGN_Amgen_ret_1d", "INTC_ret_1d", "HD_ret_1d"], "is_new": true}, {"model_id": "new_h2_NORMAL_GradientBoosting_N15_t0", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 2, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EFFR_ret_1d", "BTI_BritishAmerican_ret_5d", "EWY_Korea_zscore_60d", "BDX_Becton_Dickinson_ret_20d", "LMT_LockheedMartin_vol_20d", "EWL_Switzerland_vol_20d", "CLX_Clorox_vol_20d", "TGT_Target_zscore_60d", "MS_MorganStanley_ret_1d", "MSTR_Bitcoin3_ret_1d", "ASX_Australia_ret_5d", "HangSeng_HK_vol_20d", "SBUX_vol_20d"], "is_new": true}, {"model_id": "new_h2_NORMAL_GradientBoosting_N15_t1", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 2, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "US1Y_Rate_ret_20d", "VRP_ma5", "CMCSA_ret_1d", "EWY_Korea_zscore_60d", "PCAR_PaccarInc_ret_5d", "SJM_JM_Smucker_ret_5d", "Industrial_Production_zscore_60d", "ENB_EnbridgeInc_ret_1d", "EXC_Exelon_zscore_60d", "HangSeng_HK_ret_1d", "HangSeng_HK_ret_5d", "DAX_Germany_zscore_60d", "3M_ret_5d"], "is_new": true}, {"model_id": "new_h2_NORMAL_GradientBoosting_N15_t2", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 2, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "CPB_CampbellSoup_vol_20d", "AMGN_Amgen_ret_1d", "ENB_EnbridgeInc_ret_1d", "EWM_Malaysia_ret_1d", "SLB_Schlumberger_ret_1d", "SBUX_ret_5d", "heston_var_ev_h7", "LOW_Lowes_ret_20d", "CI_Cigna_vol_20d", "TM_Telephone_ret_1d", "NEE_NextEra_ret_20d", "NFCI_ret_5d", "IBEX_Spain_ret_20d"], "is_new": true}, {"model_id": "new_h2_NORMAL_GradientBoosting_N15_t3", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 2, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWG_Germany_vol_20d", "CMCSA_ret_1d", "LUV_SouthwestAir_ret_5d", "Brent_Oil_FRED_ret_5d", "SLB_Schlumberger_ret_1d", "EWA_Australia_ret_1d", "AMD_ret_1d", "DHR_vol_20d", "LMT_LockheedMartin_vol_20d", "SO_SouthernCo_ret_5d", "MRK_Merck_zscore_60d", "EWC_Canada_zscore_60d", "GD_GeneralDynamics_zscore_60d"], "is_new": true}, {"model_id": "new_h2_NORMAL_GradientBoosting_N15_t4", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 2, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWH_HongKong_ret_5d", "XOM_ret_1d", "TED_Spread_vol_20d", "TM_Telephone_ret_1d", "LUV_SouthwestAir_ret_5d", "heston_var_ev_h3", "ENB_EnbridgeInc_ret_1d", "Core_PCE_zscore_60d", "SBUX_zscore_60d", "T_ret_1d", "XLB_Materials_zscore_60d", "IWM_SmallCap_vol_20d", "JNJ_ret_1d"], "is_new": true}, {"model_id": "new_h2_NORMAL_GradientBoosting_N15_t5", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 2, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EOG_EOGResources_ret_5d", "US1Y_Rate_ret_20d", "MSTR_Bitcoin3_ret_1d", "IYM_BasicMaterials_ret_20d", "LOW_Lowes_ret_5d", "SPY_zscore_60d", "CTAS_Cintas_vol_20d", "PFE_ret_1d", "LLY_zscore_60d", "XLY_Disc_vol_20d", "EQIX_Equinix_ret_5d", "EWL_Switzerland_vol_20d", "TM_Telephone_ret_1d"], "is_new": true}, {"model_id": "new_h2_NORMAL_GradientBoosting_N15_t6", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 2, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "TGT_Target_zscore_60d", "US3Y_Rate_ret_5d", "CMCSA_ret_1d", "T_ret_1d", "AVB_AvalonBay_zscore_60d", "BDX_Becton_Dickinson_ret_20d", "EWC_Canada_zscore_60d", "BTI_BritishAmerican_ret_20d", "EXC_Exelon_zscore_60d", "MSTR_Bitcoin3_ret_5d", "EWA_Australia_ret_1d", "DAX_Germany_zscore_60d", "XLF_Fin_vol_20d"], "is_new": true}, {"model_id": "new_h2_NORMAL_GradientBoosting_N15_t7", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 2, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "XLF_Fin_vol_20d", "MO_AltriaMG_ret_1d", "US5Y_Rate_ret_5d", "PG_ret_20d", "EWQ_France_zscore_60d", "BA_ret_1d", "VOD_Vodafone_zscore_60d", "CPB_CampbellSoup_vol_20d", "EOG_EOGResources_vol_20d", "BTI_BritishAmerican_ret_20d", "SO_SouthernCo_ret_5d", "INTC_ret_1d", "EFFR_ret_1d"], "is_new": true}, {"model_id": "new_h2_NORMAL_GradientBoosting_N20_t0", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 2, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "CLX_Clorox_vol_20d", "HUM_Humana_ret_5d", "M_Macys_vol_20d", "EWC_Canada_zscore_60d", "EWY_Korea_ret_20d", "CI_Cigna_vol_20d", "IYR_US_REIT2_zscore_60d", "heston_var_ev_h5", "LOW_Lowes_ret_20d", "LOW_Lowes_ret_5d", "BA_ret_1d", "BTI_BritishAmerican_ret_20d", "ITT_ITTInc_ret_5d", "EWM_Malaysia_ret_1d", "ASX_Australia_vol_20d", "EWA_Australia_ret_1d", "BLK_BlackRock_zscore_60d", "HangSeng_HK_vol_20d"], "is_new": true}, {"model_id": "new_h2_NORMAL_GradientBoosting_N20_t1", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 2, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "TGT_Target_zscore_60d", "CLX_Clorox_vol_20d", "vix_mean_abs_ret_5d", "CPB_CampbellSoup_ret_5d", "AXP_Amex_vol_20d", "XLV_Health_zscore_60d", "NEE_NextEra_ret_20d", "LOW_Lowes_ret_5d", "T_ret_1d", "HangSeng_HK_ret_5d", "spx_vol_5d", "INTC_ret_1d", "INTC_ret_5d", "PCAR_PaccarInc_ret_5d", "Nikkei_Japan_zscore_60d", "EWQ_France_ret_20d", "HangSeng_HK_vol_20d", "IWM_SmallCap_vol_20d"], "is_new": true}, {"model_id": "new_h2_NORMAL_GradientBoosting_N20_t2", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 2, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "DHR_vol_20d", "NEE_NextEra_ret_20d", "XLV_Health_zscore_60d", "WTI_Oil_FRED_zscore_60d", "GD_GeneralDynamics_zscore_60d", "JNJ_ret_1d", "EFFR_ret_1d", "BDX_Becton_Dickinson_ret_20d", "NOC_Northrop_ret_20d", "HangSeng_HK_vol_20d", "CI_Cigna_vol_20d", "XLY_Disc_vol_20d", "CPB_CampbellSoup_vol_20d", "AMT_AmericanTower_ret_1d", "NVDA_vol_20d", "MSTR_Bitcoin3_ret_20d", "heston_var_ev_h5", "CMCSA_ret_1d"], "is_new": true}, {"model_id": "new_h2_NORMAL_GradientBoosting_N20_t3", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 2, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "vix_acceleration_1d", "EWY_Korea_zscore_60d", "DE_Deere_vol_20d", "DAX_Germany_vol_20d", "Michigan_Sentiment_ret_20d", "TM_Telephone_ret_1d", "DHR_vol_20d", "NOC_Northrop_ret_20d", "SJM_JM_Smucker_ret_5d", "DAX_Germany_zscore_60d", "DOW_Price_zscore_60d", "CI_Cigna_vol_20d", "heston_ev_h3", "SJM_JM_Smucker_ret_1d", "LOW_Lowes_ret_20d", "heston_var_ev_h3", "M_Macys_vol_20d", "EOG_EOGResources_ret_5d"], "is_new": true}, {"model_id": "new_h2_NORMAL_GradientBoosting_N20_t4", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 2, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "AORD_AUS_zscore_60d", "TED_Spread_zscore_60d", "EWA_Australia_zscore_60d", "gjr_condvar_h1", "GE_ret_1d", "SJM_JM_Smucker_ret_1d", "US3Y_Rate_ret_5d", "CLX_Clorox_vol_20d", "3M_ret_5d", "AMD_ret_1d", "LLY_zscore_60d", "US3M_Rate_zscore_60d", "EWM_Malaysia_vol_20d", "DHR_ret_1d", "spx_vol_5d", "heston_ev_h3", "LMT_LockheedMartin_vol_20d", "SCHW_Schwab_ret_5d"], "is_new": true}, {"model_id": "new_h2_NORMAL_GradientBoosting_N20_t5", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 2, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWG_Germany_ret_20d", "EFFR_vol_20d", "DIS_vol_20d", "AORD_AUS_zscore_60d", "AXP_Amex_ret_20d", "heston_var_ev_h7", "spx_vol_5d", "PG_ret_20d", "GD_GeneralDynamics_zscore_60d", "US30Y_Rate_ret_20d", "Industrial_Production_zscore_60d", "XLV_Health_zscore_60d", "SCHW_Schwab_ret_5d", "DAX_Germany_zscore_60d", "EXC_Exelon_ret_1d", "EWA_Australia_ret_1d", "US1Y_Rate_ret_5d", "DAX_Germany_vol_20d"], "is_new": true}, {"model_id": "new_h2_NORMAL_GradientBoosting_N20_t6", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 2, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "IYR_US_REIT2_zscore_60d", "ES_Evergy_ret_1d", "AXP_Amex_ret_20d", "IYM_BasicMaterials_ret_20d", "SO_SouthernCo_ret_5d", "heston_var_ev_h5", "AMT_AmericanTower_ret_1d", "DIS_vol_20d", "gjr_condvar_h1", "SPY_zscore_60d", "M_Macys_vol_20d", "AMGN_Amgen_ret_1d", "MSTR_Bitcoin3_ret_20d", "HD_ret_5d", "US1Y_Rate_ret_20d", "AORD_AUS_zscore_60d", "EOG_EOGResources_vol_20d", "EQR_Equity_ret_1d"], "is_new": true}, {"model_id": "new_h2_NORMAL_GradientBoosting_N20_t7", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 2, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "HangSeng_HK_ret_1d", "Retail_Sales_zscore_60d", "FedFunds_zscore_60d", "PAYX_Paychex_zscore_60d", "M_Macys_vol_20d", "XLF_Fin_vol_20d", "JNJ_ret_1d", "AVB_AvalonBay_zscore_60d", "US6M_Rate_ret_20d", "AMT_AmericanTower_ret_1d", "EXC_Exelon_ret_1d", "CI_Cigna_vol_20d", "heston_ev_h3", "SJM_JM_Smucker_ret_5d", "US7Y_Rate_ret_20d", "SLB_Schlumberger_ret_1d", "IYM_BasicMaterials_ret_20d", "ORCL_vol_20d"], "is_new": true}, {"model_id": "new_h2_NORMAL_GradientBoosting_N25_t0", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 2, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "CPB_CampbellSoup_ret_20d", "MS_MorganStanley_ret_1d", "IYM_BasicMaterials_ret_20d", "XOM_ret_20d", "LMT_LockheedMartin_vol_20d", "Nikkei_Japan_zscore_60d", "DE_Deere_vol_20d", "EWL_Switzerland_zscore_60d", "SO_SouthernCo_ret_5d", "US30Y_Rate_ret_20d", "SBUX_zscore_60d", "TED_Spread_zscore_60d", "SLB_Schlumberger_ret_5d", "XOM_ret_1d", "PAYX_Paychex_ret_20d", "MSTR_Bitcoin3_ret_20d", "LLY_zscore_60d", "NWL_Newell_ret_20d", "EWQ_France_ret_20d", "Brent_Oil_FRED_ret_20d", "EWG_Germany_vol_20d", "vix_acceleration_1d", "BA_ret_1d"], "is_new": true}, {"model_id": "new_h2_NORMAL_GradientBoosting_N25_t1", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 2, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "CLX_Clorox_vol_20d", "CPB_CampbellSoup_ret_5d", "EWA_Australia_zscore_60d", "US1Y_Rate_ret_5d", "MSTR_Bitcoin3_ret_20d", "AMD_ret_1d", "HD_ret_1d", "SBUX_ret_5d", "VOD_Vodafone_zscore_60d", "GILD_Gilead_ret_20d", "PAYX_Paychex_vol_20d", "CCI_CrownCastle_vol_20d", "US5Y_Rate_ret_5d", "XLV_Health_zscore_60d", "AMD_ret_5d", "EWJ_Japan_vol_20d", "SLB_Schlumberger_ret_5d", "AMGN_Amgen_ret_1d", "EWH_HongKong_ret_5d", "IYM_BasicMaterials_ret_20d", "DAX_Germany_vol_20d", "EXC_Exelon_ret_1d", "T10Y2Y_Spread_ret_5d"], "is_new": true}, {"model_id": "new_h2_NORMAL_GradientBoosting_N25_t2", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 2, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "US3M_Rate_zscore_60d", "HD_zscore_60d", "HangSeng_HK_vol_20d", "Brent_Oil_FRED_ret_20d", "VOD_Vodafone_zscore_60d", "DAX_Germany_vol_20d", "EXC_Exelon_zscore_60d", "PFE_ret_1d", "US3M_Rate_vol_20d", "AXP_Amex_vol_20d", "LOW_Lowes_ret_5d", "EWM_Malaysia_zscore_60d", "spx_vol_5d", "ORCL_zscore_60d", "DAX_Germany_zscore_60d", "US30Y_Rate_ret_20d", "EWQ_France_zscore_60d", "spx_abs_ret_max_5d", "LOW_Lowes_ret_20d", "CMCSA_ret_1d", "NEE_NextEra_ret_20d", "QQQ_vol_20d", "EWL_Switzerland_vol_20d"], "is_new": true}, {"model_id": "new_h2_NORMAL_GradientBoosting_N25_t3", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 2, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "DAX_Germany_zscore_60d", "spx_momentum_3d", "MS_MorganStanley_ret_5d", "SCHW_Schwab_ret_5d", "AMD_ret_1d", "US1Y_Rate_ret_5d", "EXC_Exelon_ret_1d", "SLB_Schlumberger_ret_5d", "SPY_zscore_60d", "Michigan_Sentiment_ret_20d", "XOM_ret_20d", "TED_Spread_zscore_60d", "NFCI_ret_5d", "IYM_BasicMaterials_ret_20d", "MSTR_Bitcoin3_ret_20d", "DOW_Price_zscore_60d", "VRP_ma5", "ENB_EnbridgeInc_ret_1d", "MO_AltriaMG_ret_1d", "TGT_Target_zscore_60d", "ES_Evergy_ret_1d", "EFFR_vol_20d", "TED_Spread_vol_20d"], "is_new": true}, {"model_id": "new_h2_NORMAL_GradientBoosting_N25_t4", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 2, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EFFR_ret_1d", "EWS_Singapore_ret_5d", "MRK_Merck_zscore_60d", "EWH_HongKong_ret_5d", "AVB_AvalonBay_zscore_60d", "ORCL_vol_20d", "SLB_Schlumberger_ret_5d", "SO_SouthernCo_ret_5d", "JNJ_ret_1d", "Nikkei_Japan_vol_20d", "TM_Telephone_ret_1d", "HangSeng_HK_ret_1d", "NWL_Newell_ret_20d", "SBUX_vol_20d", "NEE_NextEra_ret_20d", "SCHW_Schwab_ret_5d", "SJM_JM_Smucker_ret_5d", "AMGN_Amgen_ret_1d", "M_Macys_vol_20d", "HD_ret_1d", "ITT_ITTInc_ret_5d", "US1Y_Rate_ret_20d", "EWL_Switzerland_zscore_60d"], "is_new": true}, {"model_id": "new_h2_NORMAL_GradientBoosting_N25_t5", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 2, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "US5Y_Rate_ret_5d", "QQQ_vol_20d", "IYR_US_REIT2_zscore_60d", "DIS_vol_20d", "CCI_CrownCastle_vol_20d", "EWM_Malaysia_zscore_60d", "LOW_Lowes_ret_20d", "heston_ev_h3", "EWG_Germany_ret_20d", "spx_abs_ret_max_5d", "Retail_Sales_zscore_60d", "ITT_ITTInc_ret_5d", "LLY_zscore_60d", "IYM_BasicMaterials_ret_20d", "WTI_Oil_FRED_zscore_60d", "EFFR_vol_20d", "SJM_JM_Smucker_ret_5d", "SBUX_zscore_60d", "GILD_Gilead_ret_20d", "TM_Telephone_ret_1d", "PFE_ret_1d", "XLV_Health_zscore_60d", "CLX_Clorox_vol_20d"], "is_new": true}, {"model_id": "new_h2_NORMAL_GradientBoosting_N25_t6", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 2, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EFFR_vol_20d", "Nikkei_Japan_vol_20d", "PAYX_Paychex_ret_20d", "ITT_ITTInc_ret_5d", "Core_PCE_zscore_60d", "hmm_p_stress", "spx_abs_ret_max_5d", "XLV_Health_zscore_60d", "XLY_Disc_vol_20d", "SCHW_Schwab_ret_5d", "NWL_Newell_ret_20d", "EXC_Exelon_ret_1d", "IYR_US_REIT2_zscore_60d", "EWM_Malaysia_ret_1d", "GD_GeneralDynamics_zscore_60d", "XLK_Tech_zscore_60d", "NEE_NextEra_ret_20d", "LOW_Lowes_ret_20d", "INTC_ret_5d", "Brent_Oil_FRED_ret_20d", "XLB_Materials_zscore_60d", "ORCL_vol_20d", "IBEX_Spain_ret_20d"], "is_new": true}, {"model_id": "new_h2_NORMAL_GradientBoosting_N25_t7", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 2, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWQ_France_ret_20d", "AMD_ret_1d", "AXP_Amex_vol_20d", "US3M_Rate_vol_20d", "EWG_Germany_vol_20d", "LMT_LockheedMartin_ret_1d", "SLB_Schlumberger_ret_1d", "ES_Evergy_ret_1d", "BLK_BlackRock_zscore_60d", "CTAS_Cintas_vol_20d", "MS_MorganStanley_zscore_60d", "EWA_Australia_zscore_60d", "ASX_Australia_vol_20d", "CI_Cigna_vol_20d", "IWM_SmallCap_vol_20d", "US5Y_Rate_ret_5d", "QQQ_vol_20d", "EFFR_ret_1d", "EOG_EOGResources_vol_20d", "HD_ret_1d", "DIS_vol_20d", "DAX_Germany_zscore_60d", "TED_Spread_vol_20d"], "is_new": true}, {"model_id": "new_h2_NORMAL_GradientBoosting_N30_t0", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 2, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "Retail_Sales_zscore_60d", "MSTR_Bitcoin3_ret_20d", "NFCI_ret_5d", "US1Y_Rate_ret_20d", "AXP_Amex_vol_20d", "US3M_Rate_zscore_60d", "XOM_ret_20d", "T10Y2Y_Spread_ret_5d", "EXC_Exelon_ret_1d", "ORCL_vol_20d", "spx_abs_ret_max_5d", "CI_Cigna_vol_20d", "SLB_Schlumberger_ret_1d", "ASX_Australia_ret_5d", "GILD_Gilead_ret_20d", "AMD_ret_1d", "EWG_Germany_vol_20d", "EWC_Canada_zscore_60d", "HangSeng_HK_ret_1d", "spx_momentum_3d", "3M_vol_20d", "Brent_Oil_FRED_ret_20d", "IYR_US_REIT2_zscore_60d", "EWL_Switzerland_zscore_60d", "ORCL_zscore_60d", "EWQ_France_zscore_60d", "MRK_Merck_zscore_60d", "TED_Spread_vol_20d"], "is_new": true}, {"model_id": "new_h2_NORMAL_GradientBoosting_N30_t1", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 2, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "LLY_zscore_60d", "IWM_SmallCap_vol_20d", "EXC_Exelon_zscore_60d", "GE_ret_1d", "NOC_Northrop_ret_20d", "AMGN_Amgen_ret_1d", "vix_mean_abs_ret_5d", "EWS_Singapore_ret_5d", "US7Y_Rate_ret_20d", "EFFR_ret_1d", "MS_MorganStanley_ret_5d", "AVB_AvalonBay_zscore_60d", "XLV_Health_zscore_60d", "ORCL_vol_20d", "SBUX_vol_20d", "GD_GeneralDynamics_zscore_60d", "Michigan_Sentiment_ret_20d", "DHR_ret_1d", "SCHW_Schwab_ret_5d", "DE_Deere_vol_20d", "AXP_Amex_vol_20d", "IYR_US_REIT2_zscore_60d", "EWG_Germany_vol_20d", "MS_MorganStanley_zscore_60d", "HangSeng_HK_ret_1d", "US3Y_Rate_ret_5d", "BTI_BritishAmerican_ret_20d", "CPB_CampbellSoup_zscore_60d"], "is_new": true}, {"model_id": "new_h2_NORMAL_GradientBoosting_N30_t2", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 2, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "PAYX_Paychex_ret_20d", "XOM_ret_20d", "EFFR_ret_1d", "SJM_JM_Smucker_ret_5d", "LUV_SouthwestAir_ret_5d", "AVB_AvalonBay_zscore_60d", "EOG_EOGResources_ret_5d", "CI_Cigna_vol_20d", "LOW_Lowes_ret_20d", "ITT_ITTInc_ret_5d", "EWL_Switzerland_vol_20d", "US1Y_Rate_ret_5d", "EWS_Singapore_ret_5d", "AMT_AmericanTower_ret_1d", "SJM_JM_Smucker_ret_1d", "PCAR_PaccarInc_ret_5d", "JNJ_ret_1d", "INTC_ret_5d", "T_ret_1d", "DHR_ret_1d", "CLX_Clorox_vol_20d", "EFFR_vol_20d", "MSTR_Bitcoin3_ret_5d", "EQR_Equity_ret_1d", "NOC_Northrop_ret_20d", "EXC_Exelon_zscore_60d", "gjr_condvar_h1", "MO_AltriaMG_ret_1d"], "is_new": true}, {"model_id": "new_h2_NORMAL_GradientBoosting_N30_t3", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 2, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "MS_MorganStanley_zscore_60d", "3M_ret_5d", "PAYX_Paychex_vol_20d", "US30Y_Rate_ret_20d", "XLK_Tech_zscore_60d", "AMT_AmericanTower_ret_1d", "SBUX_ret_5d", "JNJ_ret_1d", "CI_Cigna_vol_20d", "US3M_Rate_vol_20d", "US3M_Rate_zscore_60d", "CPB_CampbellSoup_ret_5d", "XLF_Fin_vol_20d", "SLB_Schlumberger_ret_1d", "BTI_BritishAmerican_ret_20d", "PCAR_PaccarInc_ret_5d", "heston_var_ev_h7", "MRK_Merck_zscore_60d", "MO_AltriaMG_ret_1d", "BTI_BritishAmerican_ret_5d", "EFFR_ret_1d", "US3Y_Rate_ret_5d", "GILD_Gilead_ret_20d", "HangSeng_HK_ret_1d", "CPB_CampbellSoup_ret_20d", "HangSeng_HK_vol_20d", "ENB_EnbridgeInc_ret_1d", "Industrial_Production_zscore_60d"], "is_new": true}, {"model_id": "new_h2_NORMAL_GradientBoosting_N30_t4", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 2, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "XLY_Disc_vol_20d", "TM_Telephone_vol_20d", "EQIX_Equinix_ret_5d", "T10Y2Y_Spread_ret_5d", "EWS_Singapore_ret_5d", "SPY_zscore_60d", "heston_var_ev_h5", "Industrial_Production_zscore_60d", "US3Y_Rate_ret_5d", "QQQ_vol_20d", "GE_ret_1d", "ORCL_zscore_60d", "PAYX_Paychex_ret_20d", "ORCL_vol_20d", "LMT_LockheedMartin_ret_1d", "MSTR_Bitcoin3_ret_20d", "INTC_ret_5d", "US7Y_Rate_ret_20d", "ASX_Australia_vol_20d", "SBUX_ret_5d", "AMGN_Amgen_ret_1d", "LOW_Lowes_ret_5d", "IWM_SmallCap_vol_20d", "HangSeng_HK_ret_5d", "FedFunds_zscore_60d", "EWH_HongKong_ret_5d", "HangSeng_HK_vol_20d", "EWL_Switzerland_zscore_60d"], "is_new": true}, {"model_id": "new_h2_NORMAL_GradientBoosting_N30_t5", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 2, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "Industrial_Production_zscore_60d", "NFCI_ret_5d", "US3M_Rate_vol_20d", "EOG_EOGResources_vol_20d", "DHR_vol_20d", "3M_ret_5d", "US7Y_Rate_ret_20d", "GD_GeneralDynamics_zscore_60d", "XLK_Tech_zscore_60d", "BA_ret_1d", "SBUX_vol_20d", "Michigan_Sentiment_ret_20d", "EWQ_France_zscore_60d", "HangSeng_HK_ret_1d", "CTAS_Cintas_vol_20d", "LLY_zscore_60d", "EWH_HongKong_ret_5d", "LOW_Lowes_ret_5d", "VRP_ma5", "PG_ret_20d", "PLD_Prologis_ret_5d", "DAX_Germany_vol_20d", "heston_ev_h3", "AXP_Amex_vol_20d", "XLV_Health_zscore_60d", "VOD_Vodafone_zscore_60d", "IYR_US_REIT2_zscore_60d", "EWY_Korea_zscore_60d"], "is_new": true}, {"model_id": "new_h2_NORMAL_GradientBoosting_N30_t6", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 2, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "XLV_Health_zscore_60d", "FedFunds_zscore_60d", "EWQ_France_zscore_60d", "Brent_Oil_FRED_ret_20d", "vix_acceleration_1d", "TM_Telephone_ret_1d", "SBUX_ret_5d", "DE_Deere_vol_20d", "EOG_EOGResources_vol_20d", "LMT_LockheedMartin_vol_20d", "LUV_SouthwestAir_ret_5d", "PLD_Prologis_ret_5d", "SJM_JM_Smucker_ret_5d", "Industrial_Production_zscore_60d", "M_Macys_vol_20d", "EXC_Exelon_ret_1d", "CPB_CampbellSoup_vol_20d", "EWM_Malaysia_ret_1d", "JNJ_ret_1d", "GE_ret_1d", "SBUX_vol_20d", "IBEX_Spain_ret_20d", "CPB_CampbellSoup_ret_20d", "MS_MorganStanley_ret_5d", "TGT_Target_zscore_60d", "IYM_BasicMaterials_ret_20d", "MRK_Merck_zscore_60d", "XOM_ret_20d"], "is_new": true}, {"model_id": "new_h2_NORMAL_GradientBoosting_N30_t7", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 2, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "DIS_vol_20d", "TM_Telephone_ret_1d", "LUV_SouthwestAir_ret_5d", "MRK_Merck_zscore_60d", "LMT_LockheedMartin_ret_1d", "US1Y_Rate_ret_20d", "AMGN_Amgen_ret_1d", "US7Y_Rate_ret_20d", "LLY_zscore_60d", "SBUX_vol_20d", "WTI_Oil_FRED_zscore_60d", "HangSeng_HK_ret_5d", "heston_ev_h3", "LOW_Lowes_ret_5d", "SJM_JM_Smucker_ret_5d", "CPB_CampbellSoup_ret_5d", "NEE_NextEra_ret_20d", "Michigan_Sentiment_ret_20d", "AXP_Amex_ret_20d", "EFFR_ret_1d", "TED_Spread_zscore_60d", "FedFunds_zscore_60d", "Core_PCE_zscore_60d", "EWA_Australia_zscore_60d", "NFCI_ret_5d", "ENB_EnbridgeInc_ret_1d", "EWQ_France_zscore_60d", "EWJ_Japan_vol_20d"], "is_new": true}, {"model_id": "new_h2_NORMAL_RandomForest_N5_t0", "algo": "RandomForest", "regime": "NORMAL", "horizon": 2, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "HangSeng_HK_ret_1d", "AMGN_Amgen_ret_1d", "PLD_Prologis_ret_5d"], "is_new": true}, {"model_id": "new_h2_NORMAL_RandomForest_N5_t1", "algo": "RandomForest", "regime": "NORMAL", "horizon": 2, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "heston_ev_h3", "CPB_CampbellSoup_ret_20d", "EXC_Exelon_ret_1d"], "is_new": true}, {"model_id": "new_h2_NORMAL_RandomForest_N5_t2", "algo": "RandomForest", "regime": "NORMAL", "horizon": 2, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "HD_zscore_60d", "DAX_Germany_vol_20d", "US1Y_Rate_ret_5d"], "is_new": true}, {"model_id": "new_h2_NORMAL_RandomForest_N5_t3", "algo": "RandomForest", "regime": "NORMAL", "horizon": 2, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "SPY_zscore_60d", "EWL_Switzerland_vol_20d", "SJM_JM_Smucker_ret_1d"], "is_new": true}, {"model_id": "new_h2_NORMAL_RandomForest_N5_t4", "algo": "RandomForest", "regime": "NORMAL", "horizon": 2, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "DAX_Germany_vol_20d", "PG_ret_20d", "AMZN_ret_5d"], "is_new": true}, {"model_id": "new_h2_NORMAL_RandomForest_N5_t5", "algo": "RandomForest", "regime": "NORMAL", "horizon": 2, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "Brent_Oil_FRED_ret_5d", "PFE_ret_1d", "LMT_LockheedMartin_ret_1d"], "is_new": true}, {"model_id": "new_h2_NORMAL_RandomForest_N5_t6", "algo": "RandomForest", "regime": "NORMAL", "horizon": 2, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "ORCL_zscore_60d", "ASX_Australia_vol_20d", "MSTR_Bitcoin3_ret_5d"], "is_new": true}, {"model_id": "new_h2_NORMAL_RandomForest_N5_t7", "algo": "RandomForest", "regime": "NORMAL", "horizon": 2, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "TXN_vol_20d", "Nikkei_Japan_zscore_60d", "VVIX_ret_20d"], "is_new": true}, {"model_id": "new_h2_NORMAL_RandomForest_N8_t0", "algo": "RandomForest", "regime": "NORMAL", "horizon": 2, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "MS_MorganStanley_zscore_60d", "EFFR_ret_1d", "CTAS_Cintas_vol_20d", "LUV_SouthwestAir_ret_5d", "US6M_Rate_ret_20d", "EWQ_France_ret_20d"], "is_new": true}, {"model_id": "new_h2_NORMAL_RandomForest_N8_t1", "algo": "RandomForest", "regime": "NORMAL", "horizon": 2, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWY_Korea_ret_20d", "LMT_LockheedMartin_vol_20d", "EWC_Canada_zscore_60d", "Nikkei_Japan_vol_20d", "SBUX_vol_20d", "EWQ_France_zscore_60d"], "is_new": true}, {"model_id": "new_h2_NORMAL_RandomForest_N8_t2", "algo": "RandomForest", "regime": "NORMAL", "horizon": 2, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "SBUX_zscore_60d", "BDX_Becton_Dickinson_ret_20d", "PCAR_PaccarInc_ret_5d", "DAX_Germany_zscore_60d", "DE_Deere_vol_20d", "TGT_Target_zscore_60d"], "is_new": true}, {"model_id": "new_h2_NORMAL_RandomForest_N8_t3", "algo": "RandomForest", "regime": "NORMAL", "horizon": 2, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWJ_Japan_vol_20d", "EWM_Malaysia_zscore_60d", "LLY_zscore_60d", "CPB_CampbellSoup_zscore_60d", "EWA_Australia_zscore_60d", "XLK_Tech_zscore_60d"], "is_new": true}, {"model_id": "new_h2_NORMAL_RandomForest_N8_t4", "algo": "RandomForest", "regime": "NORMAL", "horizon": 2, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "AXP_Amex_vol_20d", "XLB_Materials_zscore_60d", "PLD_Prologis_ret_5d", "IBEX_Spain_ret_20d", "ENB_EnbridgeInc_ret_1d", "SCHW_Schwab_ret_5d"], "is_new": true}, {"model_id": "new_h2_NORMAL_RandomForest_N8_t5", "algo": "RandomForest", "regime": "NORMAL", "horizon": 2, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "Michigan_Sentiment_ret_20d", "EWM_Malaysia_ret_1d", "INTC_ret_1d", "Retail_Sales_zscore_60d", "US3M_Rate_zscore_60d", "NFCI_ret_5d"], "is_new": true}, {"model_id": "new_h2_NORMAL_RandomForest_N8_t6", "algo": "RandomForest", "regime": "NORMAL", "horizon": 2, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWC_Canada_zscore_60d", "vix_acceleration_1d", "AMD_ret_1d", "BA_ret_1d", "PCAR_PaccarInc_ret_5d", "EWM_Malaysia_vol_20d"], "is_new": true}, {"model_id": "new_h2_NORMAL_RandomForest_N8_t7", "algo": "RandomForest", "regime": "NORMAL", "horizon": 2, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "MS_MorganStanley_ret_1d", "AMZN_ret_5d", "SO_SouthernCo_ret_5d", "EOG_EOGResources_vol_20d", "JNJ_ret_1d", "EQR_Equity_ret_1d"], "is_new": true}, {"model_id": "new_h2_NORMAL_RandomForest_N10_t0", "algo": "RandomForest", "regime": "NORMAL", "horizon": 2, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "VRP_ma5", "FedFunds_zscore_60d", "BTI_BritishAmerican_ret_5d", "DHR_vol_20d", "SLB_Schlumberger_ret_1d", "T10Y2Y_Spread_ret_5d", "MS_MorganStanley_ret_1d", "Nikkei_Japan_vol_20d"], "is_new": true}, {"model_id": "new_h2_NORMAL_RandomForest_N10_t1", "algo": "RandomForest", "regime": "NORMAL", "horizon": 2, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "DAX_Germany_zscore_60d", "EWC_Canada_zscore_60d", "ENB_EnbridgeInc_ret_1d", "Brent_Oil_FRED_ret_5d", "FedFunds_zscore_60d", "XOM_ret_20d", "spx_momentum_3d", "ITT_ITTInc_ret_5d"], "is_new": true}, {"model_id": "new_h2_NORMAL_RandomForest_N10_t2", "algo": "RandomForest", "regime": "NORMAL", "horizon": 2, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "US7Y_Rate_ret_20d", "SBUX_zscore_60d", "heston_var_ev_h5", "TXN_vol_20d", "US3M_Rate_zscore_60d", "DIS_vol_20d", "IYR_US_REIT2_zscore_60d", "XLY_Disc_vol_20d"], "is_new": true}, {"model_id": "new_h2_NORMAL_RandomForest_N10_t3", "algo": "RandomForest", "regime": "NORMAL", "horizon": 2, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "FedFunds_zscore_60d", "gjr_condvar_h1", "GE_ret_1d", "MSTR_Bitcoin3_ret_1d", "MRK_Merck_zscore_60d", "EWJ_Japan_vol_20d", "EQR_Equity_ret_1d", "INTC_ret_5d"], "is_new": true}, {"model_id": "new_h2_NORMAL_RandomForest_N10_t4", "algo": "RandomForest", "regime": "NORMAL", "horizon": 2, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "T10Y2Y_Spread_ret_5d", "CPB_CampbellSoup_ret_20d", "TGT_Target_zscore_60d", "vix_acceleration_1d", "JNJ_ret_1d", "CI_Cigna_vol_20d", "EWJ_Japan_vol_20d", "MSTR_Bitcoin3_ret_20d"], "is_new": true}, {"model_id": "new_h2_NORMAL_RandomForest_N10_t5", "algo": "RandomForest", "regime": "NORMAL", "horizon": 2, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "DHR_vol_20d", "BDX_Becton_Dickinson_ret_20d", "EWG_Germany_vol_20d", "US1Y_Rate_ret_20d", "heston_var_ev_h7", "heston_var_ev_h5", "TM_Telephone_vol_20d", "CPB_CampbellSoup_ret_5d"], "is_new": true}, {"model_id": "new_h2_NORMAL_RandomForest_N10_t6", "algo": "RandomForest", "regime": "NORMAL", "horizon": 2, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "SO_SouthernCo_ret_5d", "HangSeng_HK_ret_5d", "CPB_CampbellSoup_ret_5d", "TM_Telephone_ret_1d", "HD_ret_1d", "SJM_JM_Smucker_ret_1d", "CCI_CrownCastle_vol_20d", "EXC_Exelon_zscore_60d"], "is_new": true}, {"model_id": "new_h2_NORMAL_RandomForest_N10_t7", "algo": "RandomForest", "regime": "NORMAL", "horizon": 2, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "Brent_Oil_FRED_ret_5d", "EXC_Exelon_zscore_60d", "spx_vol_5d", "T10Y2Y_Spread_ret_5d", "EWS_Singapore_ret_5d", "EWC_Canada_zscore_60d", "ITT_ITTInc_ret_5d", "GD_GeneralDynamics_zscore_60d"], "is_new": true}, {"model_id": "new_h2_NORMAL_RandomForest_N12_t0", "algo": "RandomForest", "regime": "NORMAL", "horizon": 2, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "PAYX_Paychex_ret_20d", "EOG_EOGResources_ret_5d", "TM_Telephone_vol_20d", "EWH_HongKong_ret_5d", "MS_MorganStanley_ret_1d", "AMD_ret_1d", "EWC_Canada_zscore_60d", "3M_ret_5d", "SLB_Schlumberger_ret_5d", "SBUX_ret_5d"], "is_new": true}, {"model_id": "new_h2_NORMAL_RandomForest_N12_t1", "algo": "RandomForest", "regime": "NORMAL", "horizon": 2, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "AXP_Amex_ret_20d", "US3Y_Rate_ret_5d", "AMD_ret_1d", "NVDA_vol_20d", "spx_vol_5d", "vix_acceleration_1d", "SBUX_ret_5d", "ORCL_vol_20d", "CPB_CampbellSoup_zscore_60d", "TXN_vol_20d"], "is_new": true}, {"model_id": "new_h2_NORMAL_RandomForest_N12_t2", "algo": "RandomForest", "regime": "NORMAL", "horizon": 2, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "XLB_Materials_zscore_60d", "MRK_Merck_zscore_60d", "3M_ret_5d", "SBUX_ret_5d", "NVDA_vol_20d", "spx_abs_ret_max_5d", "BTI_BritishAmerican_ret_20d", "heston_ev_h3", "NOC_Northrop_ret_20d", "ES_Evergy_ret_1d"], "is_new": true}, {"model_id": "new_h2_NORMAL_RandomForest_N12_t3", "algo": "RandomForest", "regime": "NORMAL", "horizon": 2, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "HD_ret_20d", "TXN_vol_20d", "TED_Spread_vol_20d", "SPY_zscore_60d", "MS_MorganStanley_zscore_60d", "US30Y_Rate_ret_20d", "AXP_Amex_vol_20d", "spx_momentum_3d", "HangSeng_HK_ret_1d", "CPB_CampbellSoup_vol_20d"], "is_new": true}, {"model_id": "new_h2_NORMAL_RandomForest_N12_t4", "algo": "RandomForest", "regime": "NORMAL", "horizon": 2, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWM_Malaysia_ret_1d", "vix_mean_abs_ret_5d", "EMR_Emerson_ret_20d", "MS_MorganStanley_ret_1d", "HangSeng_HK_ret_1d", "spx_abs_ret_max_5d", "IYM_BasicMaterials_ret_20d", "M_Macys_vol_20d", "EXC_Exelon_ret_1d", "XLV_Health_zscore_60d"], "is_new": true}, {"model_id": "new_h2_NORMAL_RandomForest_N12_t5", "algo": "RandomForest", "regime": "NORMAL", "horizon": 2, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "VOD_Vodafone_zscore_60d", "XLK_Tech_zscore_60d", "CI_Cigna_vol_20d", "M_Macys_vol_20d", "ASX_Australia_vol_20d", "ITT_ITTInc_ret_5d", "EWM_Malaysia_zscore_60d", "XLF_Fin_vol_20d", "AMT_AmericanTower_ret_1d", "MS_MorganStanley_ret_1d"], "is_new": true}, {"model_id": "new_h2_NORMAL_RandomForest_N12_t6", "algo": "RandomForest", "regime": "NORMAL", "horizon": 2, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWM_Malaysia_vol_20d", "HangSeng_HK_ret_1d", "DE_Deere_ret_5d", "Industrial_Production_zscore_60d", "spx_vol_5d", "Core_CPI_zscore_60d", "VVIX_ret_20d", "PAYX_Paychex_zscore_60d", "BA_ret_1d", "DAX_Germany_vol_20d"], "is_new": true}, {"model_id": "new_h2_NORMAL_RandomForest_N12_t7", "algo": "RandomForest", "regime": "NORMAL", "horizon": 2, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWA_Australia_ret_1d", "MS_MorganStanley_zscore_60d", "TXN_vol_20d", "DHR_vol_20d", "US6M_Rate_ret_20d", "MS_MorganStanley_ret_5d", "PG_ret_20d", "AMD_ret_1d", "GILD_Gilead_ret_20d", "EOG_EOGResources_vol_20d"], "is_new": true}, {"model_id": "new_h2_NORMAL_RandomForest_N15_t0", "algo": "RandomForest", "regime": "NORMAL", "horizon": 2, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "WTI_Oil_FRED_zscore_60d", "EWY_Korea_ret_20d", "heston_var_ev_h3", "MS_MorganStanley_zscore_60d", "DHR_ret_1d", "JNJ_ret_1d", "EOG_EOGResources_ret_5d", "M_Macys_vol_20d", "3M_ret_5d", "EOG_EOGResources_vol_20d", "CTAS_Cintas_vol_20d", "NWL_Newell_ret_20d", "US3Y_Rate_ret_5d"], "is_new": true}, {"model_id": "new_h2_NORMAL_RandomForest_N15_t1", "algo": "RandomForest", "regime": "NORMAL", "horizon": 2, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "GILD_Gilead_ret_20d", "SO_SouthernCo_ret_5d", "AMD_ret_1d", "DHR_vol_20d", "XLB_Materials_zscore_60d", "EWM_Malaysia_zscore_60d", "EWC_Canada_zscore_60d", "MO_AltriaMG_ret_1d", "TXN_vol_20d", "SJM_JM_Smucker_ret_5d", "HangSeng_HK_ret_1d", "EWA_Australia_zscore_60d", "XOM_ret_1d"], "is_new": true}, {"model_id": "new_h2_NORMAL_RandomForest_N15_t2", "algo": "RandomForest", "regime": "NORMAL", "horizon": 2, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "LOW_Lowes_ret_5d", "PAYX_Paychex_vol_20d", "EWQ_France_zscore_60d", "IWM_SmallCap_vol_20d", "CI_Cigna_vol_20d", "Retail_Sales_zscore_60d", "IBEX_Spain_ret_20d", "Industrial_Production_zscore_60d", "EWY_Korea_zscore_60d", "MSTR_Bitcoin3_ret_1d", "US3Y_Rate_ret_5d", "EWL_Switzerland_vol_20d", "LLY_zscore_60d"], "is_new": true}, {"model_id": "new_h2_NORMAL_RandomForest_N15_t3", "algo": "RandomForest", "regime": "NORMAL", "horizon": 2, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "HangSeng_HK_ret_5d", "AXP_Amex_vol_20d", "CLX_Clorox_vol_20d", "DOW_Price_zscore_60d", "TXN_vol_20d", "EWS_Singapore_ret_5d", "EWA_Australia_ret_1d", "US3Y_Rate_ret_5d", "IYM_BasicMaterials_ret_20d", "MO_AltriaMG_ret_1d", "SO_SouthernCo_ret_5d", "heston_ev_h3", "US7Y_Rate_ret_20d"], "is_new": true}, {"model_id": "new_h2_NORMAL_RandomForest_N15_t4", "algo": "RandomForest", "regime": "NORMAL", "horizon": 2, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "Industrial_Production_zscore_60d", "XLB_Materials_zscore_60d", "PCAR_PaccarInc_ret_5d", "CPB_CampbellSoup_ret_5d", "SCHW_Schwab_ret_5d", "CCI_CrownCastle_vol_20d", "SBUX_vol_20d", "AMZN_ret_5d", "EWA_Australia_zscore_60d", "ASX_Australia_ret_5d", "LLY_zscore_60d", "NFCI_ret_5d", "SJM_JM_Smucker_ret_5d"], "is_new": true}, {"model_id": "new_h2_NORMAL_RandomForest_N15_t5", "algo": "RandomForest", "regime": "NORMAL", "horizon": 2, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "DE_Deere_ret_5d", "EWY_Korea_zscore_60d", "MSTR_Bitcoin3_ret_20d", "3M_vol_20d", "IYM_BasicMaterials_ret_20d", "CMCSA_ret_1d", "MS_MorganStanley_ret_5d", "TM_Telephone_vol_20d", "EWM_Malaysia_zscore_60d", "heston_ev_h3", "gjr_condvar_h1", "US1Y_Rate_ret_5d", "VRP_ma5"], "is_new": true}, {"model_id": "new_h2_NORMAL_RandomForest_N15_t6", "algo": "RandomForest", "regime": "NORMAL", "horizon": 2, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "MSTR_Bitcoin3_ret_5d", "US30Y_Rate_ret_20d", "EXC_Exelon_zscore_60d", "Core_CPI_zscore_60d", "US1Y_Rate_ret_5d", "PCAR_PaccarInc_ret_5d", "Michigan_Sentiment_ret_20d", "PFE_ret_1d", "PG_ret_20d", "vix_acceleration_1d", "ORCL_vol_20d", "BLK_BlackRock_zscore_60d", "AMD_ret_1d"], "is_new": true}, {"model_id": "new_h2_NORMAL_RandomForest_N15_t7", "algo": "RandomForest", "regime": "NORMAL", "horizon": 2, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "IWM_SmallCap_vol_20d", "spx_abs_ret_max_5d", "VOD_Vodafone_zscore_60d", "PCAR_PaccarInc_ret_5d", "heston_ev_h3", "vix_mean_abs_ret_5d", "EWM_Malaysia_zscore_60d", "CCI_CrownCastle_vol_20d", "heston_var_ev_h3", "PPL_PPL_ret_1d", "EWQ_France_ret_20d", "AXP_Amex_ret_20d", "EWM_Malaysia_vol_20d"], "is_new": true}, {"model_id": "new_h2_NORMAL_RandomForest_N20_t0", "algo": "RandomForest", "regime": "NORMAL", "horizon": 2, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "PAYX_Paychex_zscore_60d", "IYR_US_REIT2_zscore_60d", "SJM_JM_Smucker_ret_1d", "FedFunds_zscore_60d", "JNJ_ret_1d", "Core_CPI_zscore_60d", "3M_ret_5d", "HangSeng_HK_ret_5d", "TM_Telephone_vol_20d", "HD_ret_1d", "TGT_Target_zscore_60d", "AXP_Amex_ret_20d", "VOD_Vodafone_zscore_60d", "ORCL_zscore_60d", "EWL_Switzerland_vol_20d", "CMCSA_ret_1d", "LLY_zscore_60d", "MS_MorganStanley_ret_1d"], "is_new": true}, {"model_id": "new_h2_NORMAL_RandomForest_N20_t1", "algo": "RandomForest", "regime": "NORMAL", "horizon": 2, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "SBUX_ret_5d", "BA_ret_1d", "PCAR_PaccarInc_ret_5d", "VVIX_ret_20d", "US7Y_Rate_ret_20d", "LOW_Lowes_ret_5d", "US6M_Rate_ret_20d", "HangSeng_HK_ret_1d", "Retail_Sales_zscore_60d", "NFCI_ret_5d", "Core_PCE_zscore_60d", "CPB_CampbellSoup_vol_20d", "CPB_CampbellSoup_zscore_60d", "US5Y_Rate_ret_5d", "QQQ_vol_20d", "MS_MorganStanley_ret_5d", "HangSeng_HK_vol_20d", "PAYX_Paychex_zscore_60d"], "is_new": true}, {"model_id": "new_h2_NORMAL_RandomForest_N20_t2", "algo": "RandomForest", "regime": "NORMAL", "horizon": 2, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "JNJ_ret_1d", "PG_ret_20d", "DOW_Price_zscore_60d", "SO_SouthernCo_ret_5d", "US3M_Rate_vol_20d", "PCAR_PaccarInc_ret_5d", "ENB_EnbridgeInc_ret_1d", "MS_MorganStanley_ret_1d", "US5Y_Rate_ret_5d", "EWH_HongKong_ret_5d", "HUM_Humana_ret_5d", "SLB_Schlumberger_ret_1d", "Industrial_Production_zscore_60d", "GD_GeneralDynamics_zscore_60d", "AMZN_ret_5d", "EWG_Germany_vol_20d", "US6M_Rate_ret_20d", "AMD_ret_1d"], "is_new": true}, {"model_id": "new_h2_NORMAL_RandomForest_N20_t3", "algo": "RandomForest", "regime": "NORMAL", "horizon": 2, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "NOC_Northrop_ret_20d", "AMD_ret_1d", "ORCL_vol_20d", "MSTR_Bitcoin3_ret_20d", "VRP_ma5", "VVIX_ret_20d", "EQR_Equity_ret_1d", "hmm_p_stress", "NEE_NextEra_ret_20d", "BTI_BritishAmerican_ret_5d", "EWL_Switzerland_vol_20d", "SPY_zscore_60d", "DE_Deere_ret_5d", "T_ret_1d", "QQQ_vol_20d", "ASX_Australia_ret_5d", "XOM_ret_1d", "PAYX_Paychex_ret_20d"], "is_new": true}, {"model_id": "new_h2_NORMAL_RandomForest_N20_t4", "algo": "RandomForest", "regime": "NORMAL", "horizon": 2, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "PLD_Prologis_ret_5d", "AVB_AvalonBay_zscore_60d", "spx_abs_ret_max_5d", "EWG_Germany_vol_20d", "US3M_Rate_zscore_60d", "US3M_Rate_vol_20d", "NEE_NextEra_ret_20d", "DHR_vol_20d", "MSTR_Bitcoin3_ret_1d", "EWM_Malaysia_ret_1d", "SJM_JM_Smucker_ret_1d", "VOD_Vodafone_zscore_60d", "ES_Evergy_ret_1d", "PAYX_Paychex_ret_20d", "BTI_BritishAmerican_ret_5d", "vix_mean_abs_ret_5d", "CI_Cigna_vol_20d", "BDX_Becton_Dickinson_ret_20d"], "is_new": true}, {"model_id": "new_h2_NORMAL_RandomForest_N20_t5", "algo": "RandomForest", "regime": "NORMAL", "horizon": 2, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "DE_Deere_ret_5d", "US1Y_Rate_ret_20d", "VVIX_ret_20d", "NVDA_vol_20d", "EOG_EOGResources_vol_20d", "ES_Evergy_ret_1d", "spx_vol_5d", "ASX_Australia_ret_5d", "NEE_NextEra_ret_20d", "TM_Telephone_vol_20d", "spx_abs_ret_max_5d", "ENB_EnbridgeInc_ret_1d", "HD_ret_5d", "EWG_Germany_ret_20d", "EMR_Emerson_ret_20d", "EWM_Malaysia_zscore_60d", "EWA_Australia_zscore_60d", "EWQ_France_zscore_60d"], "is_new": true}, {"model_id": "new_h2_NORMAL_RandomForest_N20_t6", "algo": "RandomForest", "regime": "NORMAL", "horizon": 2, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWY_Korea_zscore_60d", "HD_ret_1d", "MSTR_Bitcoin3_ret_20d", "AMT_AmericanTower_ret_1d", "M_Macys_vol_20d", "AORD_AUS_zscore_60d", "MSTR_Bitcoin3_ret_1d", "US3M_Rate_vol_20d", "EXC_Exelon_zscore_60d", "EWM_Malaysia_vol_20d", "3M_ret_5d", "XLK_Tech_zscore_60d", "LLY_zscore_60d", "spx_abs_ret_max_5d", "DE_Deere_ret_5d", "EWA_Australia_ret_1d", "JNJ_ret_1d", "ORCL_zscore_60d"], "is_new": true}, {"model_id": "new_h2_NORMAL_RandomForest_N20_t7", "algo": "RandomForest", "regime": "NORMAL", "horizon": 2, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "WTI_Oil_FRED_zscore_60d", "HangSeng_HK_vol_20d", "CMCSA_ret_1d", "DE_Deere_vol_20d", "SLB_Schlumberger_ret_1d", "JNJ_ret_1d", "EWS_Singapore_ret_5d", "SBUX_ret_5d", "AXP_Amex_vol_20d", "ASX_Australia_ret_5d", "MSTR_Bitcoin3_ret_20d", "FedFunds_zscore_60d", "EWY_Korea_zscore_60d", "TED_Spread_zscore_60d", "US1Y_Rate_ret_5d", "Industrial_Production_zscore_60d", "HD_ret_5d", "TGT_Target_zscore_60d"], "is_new": true}, {"model_id": "new_h2_NORMAL_RandomForest_N25_t0", "algo": "RandomForest", "regime": "NORMAL", "horizon": 2, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "Core_CPI_zscore_60d", "US6M_Rate_ret_20d", "gjr_condvar_h1", "T10Y2Y_Spread_ret_5d", "AMD_ret_1d", "EXC_Exelon_ret_1d", "ENB_EnbridgeInc_ret_1d", "LOW_Lowes_ret_5d", "LMT_LockheedMartin_ret_1d", "IBEX_Spain_ret_20d", "DHR_vol_20d", "TED_Spread_zscore_60d", "3M_vol_20d", "ITT_ITTInc_ret_5d", "EWY_Korea_ret_20d", "PAYX_Paychex_ret_20d", "NFCI_ret_5d", "HangSeng_HK_ret_5d", "DE_Deere_ret_5d", "LUV_SouthwestAir_ret_5d", "IYR_US_REIT2_zscore_60d", "AORD_AUS_zscore_60d", "Retail_Sales_zscore_60d"], "is_new": true}, {"model_id": "new_h2_NORMAL_RandomForest_N25_t1", "algo": "RandomForest", "regime": "NORMAL", "horizon": 2, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "AORD_AUS_zscore_60d", "EQR_Equity_ret_1d", "AVB_AvalonBay_zscore_60d", "EWM_Malaysia_vol_20d", "VRP_ma5", "DE_Deere_ret_5d", "EOG_EOGResources_ret_5d", "DAX_Germany_vol_20d", "EWY_Korea_ret_20d", "Brent_Oil_FRED_ret_5d", "SBUX_ret_5d", "ASX_Australia_ret_5d", "CMCSA_ret_1d", "IWM_SmallCap_vol_20d", "US30Y_Rate_ret_20d", "M_Macys_vol_20d", "NOC_Northrop_ret_20d", "EWH_HongKong_ret_5d", "spx_abs_ret_max_5d", "US5Y_Rate_ret_5d", "hmm_p_stress", "TM_Telephone_vol_20d", "PPL_PPL_ret_1d"], "is_new": true}, {"model_id": "new_h2_NORMAL_RandomForest_N25_t2", "algo": "RandomForest", "regime": "NORMAL", "horizon": 2, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWG_Germany_ret_20d", "Brent_Oil_FRED_ret_5d", "EWY_Korea_zscore_60d", "EFFR_vol_20d", "IBEX_Spain_ret_20d", "US3M_Rate_zscore_60d", "LOW_Lowes_ret_5d", "NEE_NextEra_ret_20d", "EOG_EOGResources_vol_20d", "US5Y_Rate_ret_5d", "CTAS_Cintas_vol_20d", "QQQ_vol_20d", "heston_ev_h3", "MS_MorganStanley_ret_5d", "spx_abs_ret_max_5d", "HangSeng_HK_ret_1d", "Core_CPI_zscore_60d", "Nikkei_Japan_vol_20d", "AMD_ret_1d", "EWQ_France_ret_20d", "EWA_Australia_zscore_60d", "EQR_Equity_ret_1d", "TM_Telephone_vol_20d"], "is_new": true}, {"model_id": "new_h2_NORMAL_RandomForest_N25_t3", "algo": "RandomForest", "regime": "NORMAL", "horizon": 2, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "AMGN_Amgen_ret_1d", "ITT_ITTInc_ret_5d", "CCI_CrownCastle_vol_20d", "AVB_AvalonBay_zscore_60d", "EQIX_Equinix_ret_5d", "TED_Spread_vol_20d", "SBUX_vol_20d", "IBEX_Spain_ret_20d", "Industrial_Production_zscore_60d", "spx_vol_5d", "DOW_Price_zscore_60d", "AMT_AmericanTower_ret_1d", "DIS_vol_20d", "EFFR_vol_20d", "3M_ret_5d", "CI_Cigna_vol_20d", "SBUX_zscore_60d", "LOW_Lowes_ret_5d", "HangSeng_HK_ret_1d", "LUV_SouthwestAir_ret_5d", "GE_ret_1d", "BTI_BritishAmerican_ret_5d", "MS_MorganStanley_zscore_60d"], "is_new": true}, {"model_id": "new_h2_NORMAL_RandomForest_N25_t4", "algo": "RandomForest", "regime": "NORMAL", "horizon": 2, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "BDX_Becton_Dickinson_ret_20d", "GD_GeneralDynamics_zscore_60d", "Michigan_Sentiment_ret_20d", "NOC_Northrop_ret_20d", "DE_Deere_ret_5d", "PLD_Prologis_ret_5d", "INTC_ret_5d", "AMGN_Amgen_ret_1d", "CPB_CampbellSoup_ret_5d", "LOW_Lowes_ret_20d", "EOG_EOGResources_ret_5d", "spx_vol_5d", "US3Y_Rate_ret_5d", "PAYX_Paychex_vol_20d", "EWY_Korea_zscore_60d", "GILD_Gilead_ret_20d", "gjr_condvar_h1", "BA_ret_1d", "EWM_Malaysia_vol_20d", "MS_MorganStanley_zscore_60d", "vix_mean_abs_ret_5d", "SBUX_ret_5d", "TGT_Target_zscore_60d"], "is_new": true}, {"model_id": "new_h2_NORMAL_RandomForest_N25_t5", "algo": "RandomForest", "regime": "NORMAL", "horizon": 2, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "AMD_ret_5d", "US3Y_Rate_ret_5d", "MRK_Merck_zscore_60d", "SLB_Schlumberger_ret_1d", "SCHW_Schwab_ret_5d", "DHR_vol_20d", "EWQ_France_ret_20d", "US1Y_Rate_ret_5d", "EWL_Switzerland_vol_20d", "MS_MorganStanley_zscore_60d", "TGT_Target_zscore_60d", "AMD_ret_1d", "Nikkei_Japan_vol_20d", "AXP_Amex_vol_20d", "CLX_Clorox_vol_20d", "EXC_Exelon_zscore_60d", "IYR_US_REIT2_zscore_60d", "PAYX_Paychex_vol_20d", "VVIX_ret_20d", "EFFR_ret_1d", "EWM_Malaysia_vol_20d", "DE_Deere_vol_20d", "M_Macys_vol_20d"], "is_new": true}, {"model_id": "new_h2_NORMAL_RandomForest_N25_t6", "algo": "RandomForest", "regime": "NORMAL", "horizon": 2, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "SJM_JM_Smucker_ret_5d", "CPB_CampbellSoup_ret_5d", "XLY_Disc_vol_20d", "heston_ev_h3", "PFE_ret_1d", "3M_vol_20d", "CPB_CampbellSoup_ret_20d", "LUV_SouthwestAir_ret_5d", "AXP_Amex_ret_20d", "CPB_CampbellSoup_zscore_60d", "US3M_Rate_vol_20d", "Nikkei_Japan_zscore_60d", "SJM_JM_Smucker_ret_1d", "XLF_Fin_vol_20d", "SBUX_ret_5d", "DE_Deere_ret_5d", "AORD_AUS_zscore_60d", "XLK_Tech_zscore_60d", "LMT_LockheedMartin_ret_1d", "Brent_Oil_FRED_ret_20d", "US3Y_Rate_ret_5d", "XLB_Materials_zscore_60d", "heston_var_ev_h3"], "is_new": true}, {"model_id": "new_h2_NORMAL_RandomForest_N25_t7", "algo": "RandomForest", "regime": "NORMAL", "horizon": 2, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWQ_France_ret_20d", "EWM_Malaysia_vol_20d", "DOW_Price_zscore_60d", "US3M_Rate_vol_20d", "PAYX_Paychex_zscore_60d", "3M_ret_5d", "GD_GeneralDynamics_zscore_60d", "US5Y_Rate_ret_5d", "EWC_Canada_zscore_60d", "Retail_Sales_zscore_60d", "US7Y_Rate_ret_20d", "JNJ_ret_1d", "SLB_Schlumberger_ret_1d", "EMR_Emerson_ret_20d", "VRP_ma5", "CPB_CampbellSoup_vol_20d", "XOM_ret_20d", "TM_Telephone_ret_1d", "MRK_Merck_zscore_60d", "ITT_ITTInc_ret_5d", "DAX_Germany_vol_20d", "Brent_Oil_FRED_ret_5d", "Nikkei_Japan_vol_20d"], "is_new": true}, {"model_id": "new_h2_NORMAL_RandomForest_N30_t0", "algo": "RandomForest", "regime": "NORMAL", "horizon": 2, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "WTI_Oil_FRED_zscore_60d", "ENB_EnbridgeInc_ret_1d", "AMGN_Amgen_ret_1d", "PLD_Prologis_ret_5d", "IYR_US_REIT2_zscore_60d", "PFE_ret_1d", "HangSeng_HK_ret_5d", "CTAS_Cintas_vol_20d", "IBEX_Spain_ret_20d", "EQR_Equity_ret_1d", "HangSeng_HK_ret_1d", "US30Y_Rate_ret_20d", "spx_momentum_3d", "EWQ_France_ret_20d", "US7Y_Rate_ret_20d", "NFCI_ret_5d", "EFFR_vol_20d", "MSTR_Bitcoin3_ret_5d", "DE_Deere_ret_5d", "LLY_zscore_60d", "CCI_CrownCastle_vol_20d", "SPY_zscore_60d", "TXN_vol_20d", "XLY_Disc_vol_20d", "BTI_BritishAmerican_ret_5d", "EQIX_Equinix_ret_5d", "gjr_condvar_h1", "PPL_PPL_ret_1d"], "is_new": true}, {"model_id": "new_h2_NORMAL_RandomForest_N30_t1", "algo": "RandomForest", "regime": "NORMAL", "horizon": 2, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "PG_ret_20d", "Retail_Sales_zscore_60d", "EOG_EOGResources_ret_5d", "EXC_Exelon_zscore_60d", "HangSeng_HK_ret_1d", "EWA_Australia_ret_1d", "heston_ev_h3", "HangSeng_HK_vol_20d", "EWC_Canada_zscore_60d", "gjr_condvar_h1", "JNJ_ret_1d", "HangSeng_HK_ret_5d", "XOM_ret_1d", "vix_mean_abs_ret_5d", "SPY_zscore_60d", "SCHW_Schwab_ret_5d", "ENB_EnbridgeInc_ret_1d", "XLB_Materials_zscore_60d", "MSTR_Bitcoin3_ret_1d", "LOW_Lowes_ret_5d", "QQQ_vol_20d", "CTAS_Cintas_vol_20d", "Industrial_Production_zscore_60d", "Michigan_Sentiment_ret_20d", "heston_var_ev_h5", "EWM_Malaysia_ret_1d", "XLV_Health_zscore_60d", "AXP_Amex_ret_20d"], "is_new": true}, {"model_id": "new_h2_NORMAL_RandomForest_N30_t2", "algo": "RandomForest", "regime": "NORMAL", "horizon": 2, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "TGT_Target_zscore_60d", "heston_var_ev_h7", "AMGN_Amgen_ret_1d", "heston_var_ev_h5", "HUM_Humana_ret_5d", "DAX_Germany_zscore_60d", "spx_abs_ret_max_5d", "PPL_PPL_ret_1d", "US6M_Rate_ret_20d", "NVDA_vol_20d", "AXP_Amex_vol_20d", "XOM_ret_20d", "AORD_AUS_zscore_60d", "gjr_condvar_h1", "US3M_Rate_zscore_60d", "HD_ret_5d", "AMD_ret_1d", "WTI_Oil_FRED_zscore_60d", "BTI_BritishAmerican_ret_20d", "heston_ev_h3", "LMT_LockheedMartin_vol_20d", "SBUX_ret_5d", "EWM_Malaysia_vol_20d", "Retail_Sales_zscore_60d", "NOC_Northrop_ret_20d", "hmm_p_stress", "XLK_Tech_zscore_60d", "ENB_EnbridgeInc_ret_1d"], "is_new": true}, {"model_id": "new_h2_NORMAL_RandomForest_N30_t3", "algo": "RandomForest", "regime": "NORMAL", "horizon": 2, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "LUV_SouthwestAir_ret_5d", "EXC_Exelon_zscore_60d", "VVIX_ret_20d", "EFFR_ret_1d", "ASX_Australia_ret_5d", "EWL_Switzerland_zscore_60d", "NFCI_ret_5d", "WTI_Oil_FRED_zscore_60d", "EWM_Malaysia_zscore_60d", "BA_ret_1d", "spx_vol_5d", "CI_Cigna_vol_20d", "GD_GeneralDynamics_zscore_60d", "BLK_BlackRock_zscore_60d", "CPB_CampbellSoup_ret_5d", "IYR_US_REIT2_zscore_60d", "DIS_vol_20d", "PLD_Prologis_ret_5d", "EWQ_France_zscore_60d", "CPB_CampbellSoup_zscore_60d", "Core_CPI_zscore_60d", "PFE_ret_1d", "ES_Evergy_ret_1d", "EQR_Equity_ret_1d", "MS_MorganStanley_zscore_60d", "IBEX_Spain_ret_20d", "BTI_BritishAmerican_ret_20d", "vix_acceleration_1d"], "is_new": true}, {"model_id": "new_h2_NORMAL_RandomForest_N30_t4", "algo": "RandomForest", "regime": "NORMAL", "horizon": 2, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "AMD_ret_1d", "heston_ev_h3", "BTI_BritishAmerican_ret_20d", "T10Y2Y_Spread_ret_5d", "GD_GeneralDynamics_zscore_60d", "DAX_Germany_zscore_60d", "XLK_Tech_zscore_60d", "EWM_Malaysia_ret_1d", "EWL_Switzerland_vol_20d", "IYM_BasicMaterials_ret_20d", "heston_var_ev_h7", "TED_Spread_zscore_60d", "GE_ret_1d", "PAYX_Paychex_vol_20d", "HangSeng_HK_vol_20d", "EMR_Emerson_ret_20d", "EFFR_vol_20d", "US5Y_Rate_ret_5d", "TM_Telephone_ret_1d", "EWS_Singapore_ret_5d", "ITT_ITTInc_ret_5d", "NOC_Northrop_ret_20d", "VOD_Vodafone_zscore_60d", "SBUX_ret_5d", "Retail_Sales_zscore_60d", "CI_Cigna_vol_20d", "LMT_LockheedMartin_vol_20d", "EWY_Korea_ret_20d"], "is_new": true}, {"model_id": "new_h2_NORMAL_RandomForest_N30_t5", "algo": "RandomForest", "regime": "NORMAL", "horizon": 2, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "Nikkei_Japan_zscore_60d", "EMR_Emerson_ret_20d", "HangSeng_HK_ret_5d", "PLD_Prologis_ret_5d", "EWQ_France_ret_20d", "Michigan_Sentiment_ret_20d", "IWM_SmallCap_vol_20d", "AMD_ret_1d", "AVB_AvalonBay_zscore_60d", "EWG_Germany_ret_20d", "DHR_vol_20d", "XLB_Materials_zscore_60d", "US3M_Rate_vol_20d", "AMGN_Amgen_ret_1d", "US5Y_Rate_ret_5d", "ASX_Australia_vol_20d", "BDX_Becton_Dickinson_ret_20d", "vix_mean_abs_ret_5d", "HangSeng_HK_ret_1d", "PCAR_PaccarInc_ret_5d", "NEE_NextEra_ret_20d", "BTI_BritishAmerican_ret_5d", "IBEX_Spain_ret_20d", "spx_vol_5d", "DIS_vol_20d", "PAYX_Paychex_zscore_60d", "SJM_JM_Smucker_ret_5d", "SBUX_zscore_60d"], "is_new": true}, {"model_id": "new_h2_NORMAL_RandomForest_N30_t6", "algo": "RandomForest", "regime": "NORMAL", "horizon": 2, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "spx_momentum_3d", "BA_ret_1d", "HD_ret_20d", "EQIX_Equinix_ret_5d", "heston_var_ev_h5", "EWY_Korea_ret_20d", "DAX_Germany_vol_20d", "M_Macys_vol_20d", "XLY_Disc_vol_20d", "SCHW_Schwab_ret_5d", "WTI_Oil_FRED_zscore_60d", "DHR_vol_20d", "BLK_BlackRock_zscore_60d", "Nikkei_Japan_zscore_60d", "PAYX_Paychex_ret_20d", "LUV_SouthwestAir_ret_5d", "GD_GeneralDynamics_zscore_60d", "US3Y_Rate_ret_5d", "EWM_Malaysia_ret_1d", "EWC_Canada_zscore_60d", "HangSeng_HK_vol_20d", "EWM_Malaysia_zscore_60d", "US5Y_Rate_ret_5d", "spx_abs_ret_max_5d", "AMGN_Amgen_ret_1d", "GILD_Gilead_ret_20d", "INTC_ret_5d", "EWJ_Japan_vol_20d"], "is_new": true}, {"model_id": "new_h2_NORMAL_RandomForest_N30_t7", "algo": "RandomForest", "regime": "NORMAL", "horizon": 2, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "SO_SouthernCo_ret_5d", "PPL_PPL_ret_1d", "US1Y_Rate_ret_5d", "EWL_Switzerland_zscore_60d", "IYR_US_REIT2_zscore_60d", "PCAR_PaccarInc_ret_5d", "EWC_Canada_zscore_60d", "EWY_Korea_ret_20d", "SLB_Schlumberger_ret_5d", "CPB_CampbellSoup_ret_5d", "spx_momentum_3d", "heston_var_ev_h5", "AMT_AmericanTower_ret_1d", "SBUX_ret_5d", "EWM_Malaysia_zscore_60d", "GILD_Gilead_ret_20d", "JNJ_ret_1d", "XOM_ret_20d", "EWH_HongKong_ret_5d", "AXP_Amex_vol_20d", "EQR_Equity_ret_1d", "EWM_Malaysia_vol_20d", "NWL_Newell_ret_20d", "IBEX_Spain_ret_20d", "heston_ev_h3", "EQIX_Equinix_ret_5d", "EFFR_ret_1d", "HUM_Humana_ret_5d"], "is_new": true}, {"model_id": "new_h2_NORMAL_LogisticRegression_N5_t0", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 2, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWA_Australia_ret_1d", "PFE_ret_1d", "EFFR_vol_20d"], "is_new": true}, {"model_id": "new_h2_NORMAL_LogisticRegression_N5_t1", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 2, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "AXP_Amex_ret_20d", "ITT_ITTInc_ret_5d", "Brent_Oil_FRED_ret_20d"], "is_new": true}, {"model_id": "new_h2_NORMAL_LogisticRegression_N5_t2", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 2, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "DE_Deere_ret_5d", "IWM_SmallCap_vol_20d", "CPB_CampbellSoup_ret_20d"], "is_new": true}, {"model_id": "new_h2_NORMAL_LogisticRegression_N5_t3", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 2, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "LOW_Lowes_ret_20d", "IWM_SmallCap_vol_20d", "TGT_Target_zscore_60d"], "is_new": true}, {"model_id": "new_h2_NORMAL_LogisticRegression_N5_t4", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 2, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "GE_ret_1d", "DOW_Price_zscore_60d", "heston_var_ev_h7"], "is_new": true}, {"model_id": "new_h2_NORMAL_LogisticRegression_N5_t5", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 2, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "HangSeng_HK_vol_20d", "Brent_Oil_FRED_ret_20d", "CCI_CrownCastle_vol_20d"], "is_new": true}, {"model_id": "new_h2_NORMAL_LogisticRegression_N5_t6", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 2, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "CLX_Clorox_vol_20d", "DE_Deere_ret_5d", "ITT_ITTInc_ret_5d"], "is_new": true}, {"model_id": "new_h2_NORMAL_LogisticRegression_N5_t7", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 2, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "VOD_Vodafone_zscore_60d", "CMCSA_ret_1d", "EWY_Korea_ret_20d"], "is_new": true}, {"model_id": "new_h2_NORMAL_LogisticRegression_N8_t0", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 2, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWM_Malaysia_ret_1d", "IBEX_Spain_ret_20d", "XLK_Tech_zscore_60d", "EXC_Exelon_ret_1d", "BTI_BritishAmerican_ret_5d", "XLV_Health_zscore_60d"], "is_new": true}, {"model_id": "new_h2_NORMAL_LogisticRegression_N8_t1", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 2, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "SJM_JM_Smucker_ret_1d", "INTC_ret_1d", "LOW_Lowes_ret_20d", "DAX_Germany_vol_20d", "MS_MorganStanley_ret_5d", "Nikkei_Japan_zscore_60d"], "is_new": true}, {"model_id": "new_h2_NORMAL_LogisticRegression_N8_t2", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 2, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "NEE_NextEra_ret_20d", "EWS_Singapore_ret_5d", "AMGN_Amgen_ret_1d", "Nikkei_Japan_zscore_60d", "Core_PCE_zscore_60d", "NFCI_ret_5d"], "is_new": true}, {"model_id": "new_h2_NORMAL_LogisticRegression_N8_t3", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 2, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWQ_France_ret_20d", "INTC_ret_1d", "DE_Deere_ret_5d", "AMD_ret_1d", "3M_ret_5d", "PFE_ret_1d"], "is_new": true}, {"model_id": "new_h2_NORMAL_LogisticRegression_N8_t4", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 2, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "HD_zscore_60d", "EFFR_ret_1d", "BDX_Becton_Dickinson_ret_20d", "INTC_ret_1d", "XLF_Fin_vol_20d", "US30Y_Rate_ret_20d"], "is_new": true}, {"model_id": "new_h2_NORMAL_LogisticRegression_N8_t5", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 2, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "VVIX_ret_20d", "PAYX_Paychex_vol_20d", "PFE_ret_1d", "Brent_Oil_FRED_ret_5d", "HangSeng_HK_vol_20d", "EWA_Australia_ret_1d"], "is_new": true}, {"model_id": "new_h2_NORMAL_LogisticRegression_N8_t6", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 2, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "SLB_Schlumberger_ret_1d", "AVB_AvalonBay_zscore_60d", "VRP_ma5", "ASX_Australia_vol_20d", "CPB_CampbellSoup_zscore_60d", "AXP_Amex_ret_20d"], "is_new": true}, {"model_id": "new_h2_NORMAL_LogisticRegression_N8_t7", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 2, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "AXP_Amex_vol_20d", "EWL_Switzerland_vol_20d", "WTI_Oil_FRED_zscore_60d", "DOW_Price_zscore_60d", "SBUX_vol_20d", "SO_SouthernCo_ret_5d"], "is_new": true}, {"model_id": "new_h2_NORMAL_LogisticRegression_N10_t0", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 2, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "HD_ret_1d", "MS_MorganStanley_ret_5d", "spx_momentum_3d", "Core_CPI_zscore_60d", "MSTR_Bitcoin3_ret_5d", "EWS_Singapore_ret_5d", "US1Y_Rate_ret_5d", "T10Y2Y_Spread_ret_5d"], "is_new": true}, {"model_id": "new_h2_NORMAL_LogisticRegression_N10_t1", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 2, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "CLX_Clorox_vol_20d", "TED_Spread_zscore_60d", "LMT_LockheedMartin_vol_20d", "NVDA_vol_20d", "VVIX_ret_20d", "AXP_Amex_ret_20d", "LMT_LockheedMartin_ret_1d", "XOM_ret_20d"], "is_new": true}, {"model_id": "new_h2_NORMAL_LogisticRegression_N10_t2", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 2, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "US1Y_Rate_ret_5d", "TM_Telephone_ret_1d", "EMR_Emerson_ret_20d", "MSTR_Bitcoin3_ret_20d", "Retail_Sales_zscore_60d", "EQIX_Equinix_ret_5d", "TED_Spread_vol_20d", "ASX_Australia_ret_5d"], "is_new": true}, {"model_id": "new_h2_NORMAL_LogisticRegression_N10_t3", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 2, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EFFR_vol_20d", "3M_ret_5d", "heston_var_ev_h3", "US1Y_Rate_ret_20d", "CTAS_Cintas_vol_20d", "vix_mean_abs_ret_5d", "HD_zscore_60d", "PAYX_Paychex_vol_20d"], "is_new": true}, {"model_id": "new_h2_NORMAL_LogisticRegression_N10_t4", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 2, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "LMT_LockheedMartin_vol_20d", "NVDA_vol_20d", "HD_ret_20d", "AMGN_Amgen_ret_1d", "PLD_Prologis_ret_5d", "SPY_zscore_60d", "EWM_Malaysia_ret_1d", "AORD_AUS_zscore_60d"], "is_new": true}, {"model_id": "new_h2_NORMAL_LogisticRegression_N10_t5", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 2, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "XLB_Materials_zscore_60d", "PAYX_Paychex_ret_20d", "T10Y2Y_Spread_ret_5d", "WTI_Oil_FRED_zscore_60d", "US7Y_Rate_ret_20d", "XOM_ret_20d", "US3M_Rate_zscore_60d", "MRK_Merck_zscore_60d"], "is_new": true}, {"model_id": "new_h2_NORMAL_LogisticRegression_N10_t6", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 2, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "XLF_Fin_vol_20d", "T_ret_1d", "EWG_Germany_vol_20d", "NFCI_ret_5d", "PG_ret_20d", "CCI_CrownCastle_vol_20d", "EXC_Exelon_zscore_60d", "MS_MorganStanley_ret_1d"], "is_new": true}, {"model_id": "new_h2_NORMAL_LogisticRegression_N10_t7", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 2, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "MSTR_Bitcoin3_ret_20d", "TED_Spread_vol_20d", "SO_SouthernCo_ret_5d", "NOC_Northrop_ret_20d", "PAYX_Paychex_zscore_60d", "MRK_Merck_zscore_60d", "BTI_BritishAmerican_ret_5d", "US30Y_Rate_ret_20d"], "is_new": true}, {"model_id": "new_h2_NORMAL_LogisticRegression_N12_t0", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 2, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "NWL_Newell_ret_20d", "HangSeng_HK_ret_5d", "LMT_LockheedMartin_vol_20d", "Nikkei_Japan_zscore_60d", "AXP_Amex_ret_20d", "LMT_LockheedMartin_ret_1d", "vix_mean_abs_ret_5d", "XLF_Fin_vol_20d", "LUV_SouthwestAir_ret_5d", "M_Macys_vol_20d"], "is_new": true}, {"model_id": "new_h2_NORMAL_LogisticRegression_N12_t1", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 2, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "SJM_JM_Smucker_ret_5d", "AXP_Amex_vol_20d", "XOM_ret_20d", "spx_abs_ret_max_5d", "ORCL_zscore_60d", "EWH_HongKong_ret_5d", "VOD_Vodafone_zscore_60d", "DIS_vol_20d", "AMGN_Amgen_ret_1d", "ASX_Australia_ret_5d"], "is_new": true}, {"model_id": "new_h2_NORMAL_LogisticRegression_N12_t2", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 2, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "IYM_BasicMaterials_ret_20d", "EWJ_Japan_vol_20d", "EWQ_France_zscore_60d", "PPL_PPL_ret_1d", "Retail_Sales_zscore_60d", "SBUX_ret_5d", "AMGN_Amgen_ret_1d", "EWG_Germany_ret_20d", "CPB_CampbellSoup_ret_5d", "MSTR_Bitcoin3_ret_1d"], "is_new": true}, {"model_id": "new_h2_NORMAL_LogisticRegression_N12_t3", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 2, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "NOC_Northrop_ret_20d", "CPB_CampbellSoup_zscore_60d", "MRK_Merck_zscore_60d", "HD_ret_20d", "T_ret_1d", "SBUX_ret_5d", "EFFR_ret_1d", "AMD_ret_5d", "CPB_CampbellSoup_vol_20d", "DHR_vol_20d"], "is_new": true}, {"model_id": "new_h2_NORMAL_LogisticRegression_N12_t4", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 2, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "HD_zscore_60d", "Industrial_Production_zscore_60d", "ITT_ITTInc_ret_5d", "AMZN_ret_5d", "spx_momentum_3d", "BDX_Becton_Dickinson_ret_20d", "JNJ_ret_1d", "HUM_Humana_ret_5d", "VRP_ma5", "ASX_Australia_ret_5d"], "is_new": true}, {"model_id": "new_h2_NORMAL_LogisticRegression_N12_t5", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 2, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "M_Macys_vol_20d", "gjr_condvar_h1", "PFE_ret_1d", "EWQ_France_ret_20d", "T10Y2Y_Spread_ret_5d", "SLB_Schlumberger_ret_5d", "Retail_Sales_zscore_60d", "DE_Deere_vol_20d", "IBEX_Spain_ret_20d", "MSTR_Bitcoin3_ret_1d"], "is_new": true}, {"model_id": "new_h2_NORMAL_LogisticRegression_N12_t6", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 2, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWG_Germany_ret_20d", "HD_ret_20d", "EWY_Korea_ret_20d", "TED_Spread_vol_20d", "EWL_Switzerland_zscore_60d", "vix_mean_abs_ret_5d", "BLK_BlackRock_zscore_60d", "AORD_AUS_zscore_60d", "US6M_Rate_ret_20d", "CCI_CrownCastle_vol_20d"], "is_new": true}, {"model_id": "new_h2_NORMAL_LogisticRegression_N12_t7", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 2, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "XLV_Health_zscore_60d", "HangSeng_HK_vol_20d", "XLY_Disc_vol_20d", "EWM_Malaysia_ret_1d", "AMGN_Amgen_ret_1d", "SBUX_zscore_60d", "HD_ret_1d", "CPB_CampbellSoup_zscore_60d", "ORCL_zscore_60d", "US3M_Rate_zscore_60d"], "is_new": true}, {"model_id": "new_h2_NORMAL_LogisticRegression_N15_t0", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 2, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "NFCI_ret_5d", "MS_MorganStanley_ret_1d", "EFFR_ret_1d", "SO_SouthernCo_ret_5d", "XLY_Disc_vol_20d", "SJM_JM_Smucker_ret_1d", "EWA_Australia_zscore_60d", "LOW_Lowes_ret_20d", "CPB_CampbellSoup_ret_5d", "T10Y2Y_Spread_ret_5d", "XLF_Fin_vol_20d", "MSTR_Bitcoin3_ret_1d", "EFFR_vol_20d"], "is_new": true}, {"model_id": "new_h2_NORMAL_LogisticRegression_N15_t1", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 2, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "IWM_SmallCap_vol_20d", "gjr_condvar_h1", "Michigan_Sentiment_ret_20d", "BDX_Becton_Dickinson_ret_20d", "3M_vol_20d", "ENB_EnbridgeInc_ret_1d", "DAX_Germany_zscore_60d", "EWY_Korea_ret_20d", "ASX_Australia_vol_20d", "DE_Deere_ret_5d", "EWM_Malaysia_ret_1d", "HD_ret_5d", "MRK_Merck_zscore_60d"], "is_new": true}, {"model_id": "new_h2_NORMAL_LogisticRegression_N15_t2", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 2, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "ES_Evergy_ret_1d", "LOW_Lowes_ret_5d", "CCI_CrownCastle_vol_20d", "spx_momentum_3d", "MO_AltriaMG_ret_1d", "SPY_zscore_60d", "US3M_Rate_vol_20d", "AXP_Amex_ret_20d", "FedFunds_zscore_60d", "CPB_CampbellSoup_vol_20d", "gjr_condvar_h1", "ENB_EnbridgeInc_ret_1d", "SLB_Schlumberger_ret_1d"], "is_new": true}, {"model_id": "new_h2_NORMAL_LogisticRegression_N15_t3", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 2, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "Michigan_Sentiment_ret_20d", "Nikkei_Japan_zscore_60d", "AXP_Amex_ret_20d", "TM_Telephone_vol_20d", "Core_CPI_zscore_60d", "EQR_Equity_ret_1d", "EWL_Switzerland_vol_20d", "US6M_Rate_ret_20d", "DAX_Germany_zscore_60d", "SBUX_vol_20d", "TED_Spread_vol_20d", "US1Y_Rate_ret_20d", "HUM_Humana_ret_5d"], "is_new": true}, {"model_id": "new_h2_NORMAL_LogisticRegression_N15_t4", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 2, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWA_Australia_zscore_60d", "NOC_Northrop_ret_20d", "CI_Cigna_vol_20d", "IWM_SmallCap_vol_20d", "DHR_ret_1d", "QQQ_vol_20d", "MS_MorganStanley_ret_5d", "XLK_Tech_zscore_60d", "ITT_ITTInc_ret_5d", "US1Y_Rate_ret_20d", "HD_ret_20d", "HangSeng_HK_ret_5d", "EFFR_ret_1d"], "is_new": true}, {"model_id": "new_h2_NORMAL_LogisticRegression_N15_t5", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 2, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "US1Y_Rate_ret_5d", "DOW_Price_zscore_60d", "EWY_Korea_zscore_60d", "TM_Telephone_ret_1d", "TXN_vol_20d", "MS_MorganStanley_ret_5d", "EWA_Australia_ret_1d", "Brent_Oil_FRED_ret_20d", "EWS_Singapore_ret_5d", "PG_ret_20d", "EWQ_France_ret_20d", "Brent_Oil_FRED_ret_5d", "HangSeng_HK_ret_1d"], "is_new": true}, {"model_id": "new_h2_NORMAL_LogisticRegression_N15_t6", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 2, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "CTAS_Cintas_vol_20d", "TGT_Target_zscore_60d", "ASX_Australia_ret_5d", "EFFR_ret_1d", "MSTR_Bitcoin3_ret_20d", "gjr_condvar_h1", "VOD_Vodafone_zscore_60d", "DAX_Germany_vol_20d", "FedFunds_zscore_60d", "XLB_Materials_zscore_60d", "EWY_Korea_ret_20d", "vix_mean_abs_ret_5d", "US3Y_Rate_ret_5d"], "is_new": true}, {"model_id": "new_h2_NORMAL_LogisticRegression_N15_t7", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 2, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "AMZN_ret_5d", "FedFunds_zscore_60d", "PCAR_PaccarInc_ret_5d", "CMCSA_ret_1d", "SJM_JM_Smucker_ret_5d", "US3M_Rate_zscore_60d", "DAX_Germany_vol_20d", "CPB_CampbellSoup_vol_20d", "US6M_Rate_ret_20d", "XLV_Health_zscore_60d", "DAX_Germany_zscore_60d", "INTC_ret_1d", "MSTR_Bitcoin3_ret_1d"], "is_new": true}, {"model_id": "new_h2_NORMAL_LogisticRegression_N20_t0", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 2, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "NEE_NextEra_ret_20d", "US30Y_Rate_ret_20d", "US1Y_Rate_ret_20d", "vix_mean_abs_ret_5d", "EWJ_Japan_vol_20d", "HD_zscore_60d", "BTI_BritishAmerican_ret_20d", "AXP_Amex_ret_20d", "EWL_Switzerland_zscore_60d", "FedFunds_zscore_60d", "CCI_CrownCastle_vol_20d", "HD_ret_1d", "CMCSA_ret_1d", "EWC_Canada_zscore_60d", "BTI_BritishAmerican_ret_5d", "SPY_zscore_60d", "CPB_CampbellSoup_vol_20d", "CTAS_Cintas_vol_20d"], "is_new": true}, {"model_id": "new_h2_NORMAL_LogisticRegression_N20_t1", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 2, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "JNJ_ret_1d", "CI_Cigna_vol_20d", "EWM_Malaysia_zscore_60d", "EFFR_vol_20d", "INTC_ret_1d", "heston_ev_h3", "Core_CPI_zscore_60d", "PAYX_Paychex_ret_20d", "VOD_Vodafone_zscore_60d", "vix_mean_abs_ret_5d", "3M_ret_5d", "IBEX_Spain_ret_20d", "PAYX_Paychex_zscore_60d", "ASX_Australia_vol_20d", "SJM_JM_Smucker_ret_5d", "LOW_Lowes_ret_5d", "NOC_Northrop_ret_20d", "US3Y_Rate_ret_5d"], "is_new": true}, {"model_id": "new_h2_NORMAL_LogisticRegression_N20_t2", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 2, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "PCAR_PaccarInc_ret_5d", "CPB_CampbellSoup_ret_5d", "HUM_Humana_ret_5d", "spx_vol_5d", "SJM_JM_Smucker_ret_5d", "PFE_ret_1d", "TED_Spread_vol_20d", "EWJ_Japan_vol_20d", "BTI_BritishAmerican_ret_5d", "AMT_AmericanTower_ret_1d", "EWC_Canada_zscore_60d", "CPB_CampbellSoup_vol_20d", "FedFunds_zscore_60d", "LMT_LockheedMartin_vol_20d", "MS_MorganStanley_ret_1d", "GE_ret_1d", "INTC_ret_1d", "QQQ_vol_20d"], "is_new": true}, {"model_id": "new_h2_NORMAL_LogisticRegression_N20_t3", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 2, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "spx_vol_5d", "PPL_PPL_ret_1d", "WTI_Oil_FRED_zscore_60d", "GD_GeneralDynamics_zscore_60d", "BA_ret_1d", "PCAR_PaccarInc_ret_5d", "AMZN_ret_5d", "EWL_Switzerland_zscore_60d", "EWJ_Japan_vol_20d", "PAYX_Paychex_zscore_60d", "FedFunds_zscore_60d", "EXC_Exelon_zscore_60d", "XLY_Disc_vol_20d", "XOM_ret_20d", "US6M_Rate_ret_20d", "XLV_Health_zscore_60d", "TED_Spread_vol_20d", "3M_ret_5d"], "is_new": true}, {"model_id": "new_h2_NORMAL_LogisticRegression_N20_t4", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 2, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWM_Malaysia_vol_20d", "MS_MorganStanley_ret_5d", "XLV_Health_zscore_60d", "HD_ret_5d", "CLX_Clorox_vol_20d", "ES_Evergy_ret_1d", "gjr_condvar_h1", "Michigan_Sentiment_ret_20d", "HangSeng_HK_ret_1d", "Brent_Oil_FRED_ret_20d", "3M_vol_20d", "MSTR_Bitcoin3_ret_5d", "AMGN_Amgen_ret_1d", "EWM_Malaysia_ret_1d", "ENB_EnbridgeInc_ret_1d", "Nikkei_Japan_vol_20d", "PAYX_Paychex_zscore_60d", "LMT_LockheedMartin_ret_1d"], "is_new": true}, {"model_id": "new_h2_NORMAL_LogisticRegression_N20_t5", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 2, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "SBUX_zscore_60d", "MS_MorganStanley_zscore_60d", "EWQ_France_zscore_60d", "CLX_Clorox_vol_20d", "ASX_Australia_vol_20d", "CCI_CrownCastle_vol_20d", "AMZN_ret_5d", "TGT_Target_zscore_60d", "XLB_Materials_zscore_60d", "Michigan_Sentiment_ret_20d", "heston_var_ev_h5", "IYR_US_REIT2_zscore_60d", "HUM_Humana_ret_5d", "NEE_NextEra_ret_20d", "CPB_CampbellSoup_ret_20d", "ENB_EnbridgeInc_ret_1d", "ITT_ITTInc_ret_5d", "EWM_Malaysia_ret_1d"], "is_new": true}, {"model_id": "new_h2_NORMAL_LogisticRegression_N20_t6", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 2, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "AMT_AmericanTower_ret_1d", "BLK_BlackRock_zscore_60d", "HD_ret_1d", "ORCL_zscore_60d", "HD_zscore_60d", "T10Y2Y_Spread_ret_5d", "M_Macys_vol_20d", "PG_ret_20d", "WTI_Oil_FRED_zscore_60d", "HangSeng_HK_ret_1d", "US30Y_Rate_ret_20d", "MSTR_Bitcoin3_ret_1d", "heston_ev_h3", "US7Y_Rate_ret_20d", "SPY_zscore_60d", "VRP_ma5", "GE_ret_1d", "SLB_Schlumberger_ret_5d"], "is_new": true}, {"model_id": "new_h2_NORMAL_LogisticRegression_N20_t7", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 2, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "hmm_p_stress", "EWY_Korea_ret_20d", "Michigan_Sentiment_ret_20d", "CPB_CampbellSoup_zscore_60d", "AMZN_ret_5d", "BA_ret_1d", "PCAR_PaccarInc_ret_5d", "PLD_Prologis_ret_5d", "3M_vol_20d", "Brent_Oil_FRED_ret_20d", "PAYX_Paychex_ret_20d", "SBUX_vol_20d", "EWL_Switzerland_zscore_60d", "spx_abs_ret_max_5d", "SBUX_ret_5d", "INTC_ret_1d", "US3Y_Rate_ret_5d", "IBEX_Spain_ret_20d"], "is_new": true}, {"model_id": "new_h2_NORMAL_LogisticRegression_N25_t0", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 2, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "PAYX_Paychex_ret_20d", "TED_Spread_zscore_60d", "CPB_CampbellSoup_vol_20d", "CTAS_Cintas_vol_20d", "Brent_Oil_FRED_ret_20d", "vix_mean_abs_ret_5d", "CPB_CampbellSoup_zscore_60d", "US3Y_Rate_ret_5d", "NWL_Newell_ret_20d", "BA_ret_1d", "ASX_Australia_vol_20d", "HD_ret_1d", "XLV_Health_zscore_60d", "AMZN_ret_5d", "EOG_EOGResources_ret_5d", "CMCSA_ret_1d", "GD_GeneralDynamics_zscore_60d", "HD_ret_5d", "HUM_Humana_ret_5d", "CLX_Clorox_vol_20d", "LOW_Lowes_ret_20d", "IYR_US_REIT2_zscore_60d", "heston_var_ev_h3"], "is_new": true}, {"model_id": "new_h2_NORMAL_LogisticRegression_N25_t1", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 2, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWM_Malaysia_zscore_60d", "vix_acceleration_1d", "spx_vol_5d", "SPY_zscore_60d", "HangSeng_HK_ret_5d", "US3M_Rate_vol_20d", "NEE_NextEra_ret_20d", "JNJ_ret_1d", "MRK_Merck_zscore_60d", "EMR_Emerson_ret_20d", "EWM_Malaysia_vol_20d", "US3M_Rate_zscore_60d", "CPB_CampbellSoup_zscore_60d", "Retail_Sales_zscore_60d", "AMZN_ret_5d", "EWJ_Japan_vol_20d", "vix_mean_abs_ret_5d", "LOW_Lowes_ret_5d", "EWS_Singapore_ret_5d", "3M_ret_5d", "BDX_Becton_Dickinson_ret_20d", "IBEX_Spain_ret_20d", "US7Y_Rate_ret_20d"], "is_new": true}, {"model_id": "new_h2_NORMAL_LogisticRegression_N25_t2", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 2, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "LMT_LockheedMartin_ret_1d", "EWJ_Japan_vol_20d", "MSTR_Bitcoin3_ret_5d", "CTAS_Cintas_vol_20d", "XLV_Health_zscore_60d", "Retail_Sales_zscore_60d", "heston_var_ev_h7", "PCAR_PaccarInc_ret_5d", "EWQ_France_ret_20d", "CPB_CampbellSoup_vol_20d", "heston_var_ev_h5", "EWL_Switzerland_vol_20d", "US3M_Rate_zscore_60d", "HangSeng_HK_ret_5d", "Nikkei_Japan_vol_20d", "EFFR_ret_1d", "GE_ret_1d", "SLB_Schlumberger_ret_1d", "SCHW_Schwab_ret_5d", "DOW_Price_zscore_60d", "EWC_Canada_zscore_60d", "US30Y_Rate_ret_20d", "vix_mean_abs_ret_5d"], "is_new": true}, {"model_id": "new_h2_NORMAL_LogisticRegression_N25_t3", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 2, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "XOM_ret_1d", "EFFR_ret_1d", "Retail_Sales_zscore_60d", "spx_momentum_3d", "MS_MorganStanley_ret_1d", "US1Y_Rate_ret_20d", "PCAR_PaccarInc_ret_5d", "CMCSA_ret_1d", "EXC_Exelon_zscore_60d", "CCI_CrownCastle_vol_20d", "EWQ_France_zscore_60d", "GILD_Gilead_ret_20d", "US5Y_Rate_ret_5d", "EWG_Germany_ret_20d", "DAX_Germany_vol_20d", "SBUX_ret_5d", "XLF_Fin_vol_20d", "spx_abs_ret_max_5d", "ORCL_zscore_60d", "ASX_Australia_ret_5d", "Nikkei_Japan_zscore_60d", "3M_vol_20d", "LUV_SouthwestAir_ret_5d"], "is_new": true}, {"model_id": "new_h2_NORMAL_LogisticRegression_N25_t4", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 2, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "INTC_ret_5d", "PG_ret_20d", "XLF_Fin_vol_20d", "DAX_Germany_zscore_60d", "Core_PCE_zscore_60d", "DAX_Germany_vol_20d", "MS_MorganStanley_ret_1d", "DE_Deere_vol_20d", "Industrial_Production_zscore_60d", "AMZN_ret_5d", "MS_MorganStanley_ret_5d", "VOD_Vodafone_zscore_60d", "US1Y_Rate_ret_5d", "XLK_Tech_zscore_60d", "EWA_Australia_ret_1d", "MO_AltriaMG_ret_1d", "FedFunds_zscore_60d", "gjr_condvar_h1", "HangSeng_HK_ret_5d", "M_Macys_vol_20d", "EWH_HongKong_ret_5d", "TGT_Target_zscore_60d", "PAYX_Paychex_vol_20d"], "is_new": true}, {"model_id": "new_h2_NORMAL_LogisticRegression_N25_t5", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 2, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "LOW_Lowes_ret_5d", "SBUX_vol_20d", "3M_vol_20d", "EWM_Malaysia_zscore_60d", "WTI_Oil_FRED_zscore_60d", "EWQ_France_ret_20d", "EWY_Korea_zscore_60d", "DE_Deere_ret_5d", "QQQ_vol_20d", "CPB_CampbellSoup_ret_5d", "TM_Telephone_vol_20d", "MS_MorganStanley_zscore_60d", "XOM_ret_20d", "US7Y_Rate_ret_20d", "DOW_Price_zscore_60d", "vix_mean_abs_ret_5d", "ITT_ITTInc_ret_5d", "DAX_Germany_zscore_60d", "BDX_Becton_Dickinson_ret_20d", "IYM_BasicMaterials_ret_20d", "AMZN_ret_5d", "EOG_EOGResources_vol_20d", "MS_MorganStanley_ret_1d"], "is_new": true}, {"model_id": "new_h2_NORMAL_LogisticRegression_N25_t6", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 2, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "CTAS_Cintas_vol_20d", "vix_acceleration_1d", "BDX_Becton_Dickinson_ret_20d", "CI_Cigna_vol_20d", "IWM_SmallCap_vol_20d", "NWL_Newell_ret_20d", "Brent_Oil_FRED_ret_20d", "NOC_Northrop_ret_20d", "heston_var_ev_h7", "US7Y_Rate_ret_20d", "PFE_ret_1d", "EWA_Australia_zscore_60d", "US30Y_Rate_ret_20d", "EXC_Exelon_zscore_60d", "EWY_Korea_zscore_60d", "SO_SouthernCo_ret_5d", "CCI_CrownCastle_vol_20d", "DHR_ret_1d", "CLX_Clorox_vol_20d", "EWM_Malaysia_zscore_60d", "3M_ret_5d", "AXP_Amex_vol_20d", "US1Y_Rate_ret_5d"], "is_new": true}, {"model_id": "new_h2_NORMAL_LogisticRegression_N25_t7", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 2, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "vix_acceleration_1d", "MSTR_Bitcoin3_ret_1d", "US3Y_Rate_ret_5d", "XOM_ret_1d", "BA_ret_1d", "US30Y_Rate_ret_20d", "Michigan_Sentiment_ret_20d", "Retail_Sales_zscore_60d", "DAX_Germany_zscore_60d", "EWC_Canada_zscore_60d", "EWL_Switzerland_vol_20d", "Brent_Oil_FRED_ret_20d", "BLK_BlackRock_zscore_60d", "FedFunds_zscore_60d", "XLY_Disc_vol_20d", "TED_Spread_zscore_60d", "DAX_Germany_vol_20d", "SCHW_Schwab_ret_5d", "PLD_Prologis_ret_5d", "EWS_Singapore_ret_5d", "NFCI_ret_5d", "LOW_Lowes_ret_5d", "INTC_ret_5d"], "is_new": true}, {"model_id": "new_h2_NORMAL_LogisticRegression_N30_t0", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 2, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "DAX_Germany_zscore_60d", "Brent_Oil_FRED_ret_20d", "TED_Spread_zscore_60d", "T_ret_1d", "DE_Deere_vol_20d", "DHR_ret_1d", "AMT_AmericanTower_ret_1d", "NFCI_ret_5d", "EWJ_Japan_vol_20d", "CCI_CrownCastle_vol_20d", "TED_Spread_vol_20d", "MS_MorganStanley_zscore_60d", "HD_ret_5d", "DAX_Germany_vol_20d", "SBUX_ret_5d", "MSTR_Bitcoin3_ret_5d", "BTI_BritishAmerican_ret_20d", "EWM_Malaysia_ret_1d", "AORD_AUS_zscore_60d", "EFFR_ret_1d", "EWA_Australia_ret_1d", "heston_var_ev_h5", "FedFunds_zscore_60d", "Core_CPI_zscore_60d", "ORCL_vol_20d", "EWQ_France_zscore_60d", "NWL_Newell_ret_20d", "SLB_Schlumberger_ret_1d"], "is_new": true}, {"model_id": "new_h2_NORMAL_LogisticRegression_N30_t1", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 2, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "PG_ret_20d", "ORCL_vol_20d", "GILD_Gilead_ret_20d", "AMD_ret_5d", "EWS_Singapore_ret_5d", "DIS_vol_20d", "EWM_Malaysia_ret_1d", "US5Y_Rate_ret_5d", "BTI_BritishAmerican_ret_5d", "US3Y_Rate_ret_5d", "EWM_Malaysia_vol_20d", "EQIX_Equinix_ret_5d", "JNJ_ret_1d", "DHR_ret_1d", "TGT_Target_zscore_60d", "DOW_Price_zscore_60d", "M_Macys_vol_20d", "PCAR_PaccarInc_ret_5d", "CPB_CampbellSoup_vol_20d", "EOG_EOGResources_vol_20d", "MSTR_Bitcoin3_ret_20d", "EWQ_France_zscore_60d", "vix_mean_abs_ret_5d", "EXC_Exelon_zscore_60d", "US3M_Rate_zscore_60d", "ITT_ITTInc_ret_5d", "ENB_EnbridgeInc_ret_1d", "ES_Evergy_ret_1d"], "is_new": true}, {"model_id": "new_h2_NORMAL_LogisticRegression_N30_t2", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 2, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EXC_Exelon_zscore_60d", "HangSeng_HK_ret_5d", "T10Y2Y_Spread_ret_5d", "HD_ret_1d", "EWY_Korea_ret_20d", "gjr_condvar_h1", "Nikkei_Japan_vol_20d", "TM_Telephone_vol_20d", "ORCL_vol_20d", "SLB_Schlumberger_ret_1d", "CTAS_Cintas_vol_20d", "Core_CPI_zscore_60d", "Brent_Oil_FRED_ret_5d", "IBEX_Spain_ret_20d", "EWL_Switzerland_zscore_60d", "XLV_Health_zscore_60d", "AXP_Amex_ret_20d", "heston_var_ev_h7", "EXC_Exelon_ret_1d", "AMZN_ret_5d", "spx_vol_5d", "VVIX_ret_20d", "ITT_ITTInc_ret_5d", "US30Y_Rate_ret_20d", "SJM_JM_Smucker_ret_1d", "Retail_Sales_zscore_60d", "CI_Cigna_vol_20d", "IYR_US_REIT2_zscore_60d"], "is_new": true}, {"model_id": "new_h2_NORMAL_LogisticRegression_N30_t3", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 2, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "ENB_EnbridgeInc_ret_1d", "SBUX_ret_5d", "TED_Spread_zscore_60d", "Retail_Sales_zscore_60d", "XLK_Tech_zscore_60d", "BA_ret_1d", "EOG_EOGResources_ret_5d", "LOW_Lowes_ret_20d", "US5Y_Rate_ret_5d", "CPB_CampbellSoup_ret_5d", "DOW_Price_zscore_60d", "AMGN_Amgen_ret_1d", "GD_GeneralDynamics_zscore_60d", "MS_MorganStanley_ret_1d", "EWQ_France_ret_20d", "PLD_Prologis_ret_5d", "heston_var_ev_h7", "Core_CPI_zscore_60d", "NFCI_ret_5d", "EQIX_Equinix_ret_5d", "MS_MorganStanley_zscore_60d", "VRP_ma5", "AMT_AmericanTower_ret_1d", "EWL_Switzerland_vol_20d", "DIS_vol_20d", "EWA_Australia_ret_1d", "ITT_ITTInc_ret_5d", "SLB_Schlumberger_ret_5d"], "is_new": true}, {"model_id": "new_h2_NORMAL_LogisticRegression_N30_t4", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 2, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "XOM_ret_1d", "INTC_ret_5d", "LLY_zscore_60d", "spx_momentum_3d", "XLY_Disc_vol_20d", "VOD_Vodafone_zscore_60d", "US3M_Rate_zscore_60d", "SBUX_zscore_60d", "PLD_Prologis_ret_5d", "EWM_Malaysia_vol_20d", "PCAR_PaccarInc_ret_5d", "US3Y_Rate_ret_5d", "EWC_Canada_zscore_60d", "XLB_Materials_zscore_60d", "LOW_Lowes_ret_20d", "heston_var_ev_h5", "US6M_Rate_ret_20d", "INTC_ret_1d", "spx_vol_5d", "BTI_BritishAmerican_ret_5d", "US3M_Rate_vol_20d", "DIS_vol_20d", "EXC_Exelon_ret_1d", "PAYX_Paychex_zscore_60d", "GD_GeneralDynamics_zscore_60d", "CI_Cigna_vol_20d", "Brent_Oil_FRED_ret_20d", "BTI_BritishAmerican_ret_20d"], "is_new": true}, {"model_id": "new_h2_NORMAL_LogisticRegression_N30_t5", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 2, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "AMD_ret_5d", "EWL_Switzerland_vol_20d", "TED_Spread_zscore_60d", "NWL_Newell_ret_20d", "DIS_vol_20d", "LMT_LockheedMartin_vol_20d", "MS_MorganStanley_ret_5d", "PG_ret_20d", "Nikkei_Japan_zscore_60d", "CPB_CampbellSoup_vol_20d", "heston_ev_h3", "EMR_Emerson_ret_20d", "LUV_SouthwestAir_ret_5d", "US1Y_Rate_ret_5d", "HD_zscore_60d", "XOM_ret_20d", "EWG_Germany_vol_20d", "EWH_HongKong_ret_5d", "AORD_AUS_zscore_60d", "vix_mean_abs_ret_5d", "TXN_vol_20d", "AVB_AvalonBay_zscore_60d", "LOW_Lowes_ret_5d", "MS_MorganStanley_zscore_60d", "EWY_Korea_ret_20d", "ASX_Australia_ret_5d", "SLB_Schlumberger_ret_5d", "spx_abs_ret_max_5d"], "is_new": true}, {"model_id": "new_h2_NORMAL_LogisticRegression_N30_t6", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 2, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "SLB_Schlumberger_ret_5d", "PLD_Prologis_ret_5d", "EWM_Malaysia_ret_1d", "EOG_EOGResources_ret_5d", "CTAS_Cintas_vol_20d", "AVB_AvalonBay_zscore_60d", "PPL_PPL_ret_1d", "T10Y2Y_Spread_ret_5d", "TED_Spread_vol_20d", "EWG_Germany_ret_20d", "CI_Cigna_vol_20d", "spx_momentum_3d", "ORCL_zscore_60d", "EWM_Malaysia_vol_20d", "Nikkei_Japan_vol_20d", "LMT_LockheedMartin_vol_20d", "HD_zscore_60d", "TXN_vol_20d", "LMT_LockheedMartin_ret_1d", "DOW_Price_zscore_60d", "ES_Evergy_ret_1d", "3M_ret_5d", "heston_var_ev_h7", "DE_Deere_vol_20d", "3M_vol_20d", "BTI_BritishAmerican_ret_20d", "PAYX_Paychex_vol_20d", "AMT_AmericanTower_ret_1d"], "is_new": true}, {"model_id": "new_h2_NORMAL_LogisticRegression_N30_t7", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 2, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWY_Korea_zscore_60d", "BTI_BritishAmerican_ret_5d", "ORCL_vol_20d", "EMR_Emerson_ret_20d", "SBUX_zscore_60d", "XLK_Tech_zscore_60d", "Nikkei_Japan_zscore_60d", "EWH_HongKong_ret_5d", "gjr_condvar_h1", "MS_MorganStanley_zscore_60d", "IWM_SmallCap_vol_20d", "BTI_BritishAmerican_ret_20d", "EFFR_vol_20d", "heston_var_ev_h3", "LMT_LockheedMartin_vol_20d", "IYM_BasicMaterials_ret_20d", "CMCSA_ret_1d", "AMGN_Amgen_ret_1d", "XLB_Materials_zscore_60d", "LUV_SouthwestAir_ret_5d", "CCI_CrownCastle_vol_20d", "ASX_Australia_ret_5d", "SO_SouthernCo_ret_5d", "Core_CPI_zscore_60d", "HangSeng_HK_ret_5d", "TM_Telephone_ret_1d", "EXC_Exelon_ret_1d", "DAX_Germany_zscore_60d"], "is_new": true}, {"model_id": "new_h2_STRESS_XGBoost_N5_t0", "algo": "XGBoost", "regime": "STRESS", "horizon": 2, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "AXP_Amex_vol_20d", "AMD_ret_5d", "EWH_HongKong_ret_5d"], "is_new": true}, {"model_id": "new_h2_STRESS_XGBoost_N5_t1", "algo": "XGBoost", "regime": "STRESS", "horizon": 2, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "DE_Deere_ret_5d", "EWQ_France_zscore_60d", "ITT_ITTInc_ret_5d"], "is_new": true}, {"model_id": "new_h2_STRESS_XGBoost_N5_t2", "algo": "XGBoost", "regime": "STRESS", "horizon": 2, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWA_Australia_ret_1d", "vix_mean_abs_ret_5d", "GE_ret_1d"], "is_new": true}, {"model_id": "new_h2_STRESS_XGBoost_N5_t3", "algo": "XGBoost", "regime": "STRESS", "horizon": 2, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "PCAR_PaccarInc_ret_5d", "IYM_BasicMaterials_ret_20d", "EWG_Germany_vol_20d"], "is_new": true}, {"model_id": "new_h2_STRESS_XGBoost_N5_t4", "algo": "XGBoost", "regime": "STRESS", "horizon": 2, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "US30Y_Rate_ret_20d", "AORD_AUS_zscore_60d", "XLF_Fin_vol_20d"], "is_new": true}, {"model_id": "new_h2_STRESS_XGBoost_N5_t5", "algo": "XGBoost", "regime": "STRESS", "horizon": 2, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "DAX_Germany_vol_20d", "NWL_Newell_ret_20d", "Nikkei_Japan_vol_20d"], "is_new": true}, {"model_id": "new_h2_STRESS_XGBoost_N5_t6", "algo": "XGBoost", "regime": "STRESS", "horizon": 2, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "AMD_ret_1d", "EWY_Korea_ret_20d", "EWG_Germany_vol_20d"], "is_new": true}, {"model_id": "new_h2_STRESS_XGBoost_N5_t7", "algo": "XGBoost", "regime": "STRESS", "horizon": 2, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWY_Korea_zscore_60d", "BTI_BritishAmerican_ret_20d", "AMGN_Amgen_ret_1d"], "is_new": true}, {"model_id": "new_h2_STRESS_XGBoost_N8_t0", "algo": "XGBoost", "regime": "STRESS", "horizon": 2, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "FedFunds_zscore_60d", "US6M_Rate_ret_20d", "PG_ret_20d", "3M_vol_20d", "AORD_AUS_zscore_60d", "TXN_vol_20d"], "is_new": true}, {"model_id": "new_h2_STRESS_XGBoost_N8_t1", "algo": "XGBoost", "regime": "STRESS", "horizon": 2, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "CPB_CampbellSoup_ret_5d", "IYM_BasicMaterials_ret_20d", "spx_abs_ret_max_5d", "CPB_CampbellSoup_ret_20d", "SBUX_zscore_60d", "NEE_NextEra_ret_20d"], "is_new": true}, {"model_id": "new_h2_STRESS_XGBoost_N8_t2", "algo": "XGBoost", "regime": "STRESS", "horizon": 2, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "SPY_zscore_60d", "3M_ret_5d", "VOD_Vodafone_zscore_60d", "DE_Deere_ret_5d", "HUM_Humana_ret_5d", "CPB_CampbellSoup_ret_20d"], "is_new": true}, {"model_id": "new_h2_STRESS_XGBoost_N8_t3", "algo": "XGBoost", "regime": "STRESS", "horizon": 2, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "AMGN_Amgen_ret_1d", "spx_abs_ret_max_5d", "XLK_Tech_zscore_60d", "NEE_NextEra_ret_20d", "XLV_Health_zscore_60d", "US3Y_Rate_ret_5d"], "is_new": true}, {"model_id": "new_h2_STRESS_XGBoost_N8_t4", "algo": "XGBoost", "regime": "STRESS", "horizon": 2, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "DAX_Germany_vol_20d", "IYM_BasicMaterials_ret_20d", "EMR_Emerson_ret_20d", "MRK_Merck_zscore_60d", "EFFR_vol_20d", "EWS_Singapore_ret_5d"], "is_new": true}, {"model_id": "new_h2_STRESS_XGBoost_N8_t5", "algo": "XGBoost", "regime": "STRESS", "horizon": 2, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "IWM_SmallCap_vol_20d", "SJM_JM_Smucker_ret_5d", "T_ret_1d", "EWA_Australia_zscore_60d", "PCAR_PaccarInc_ret_5d", "heston_var_ev_h7"], "is_new": true}, {"model_id": "new_h2_STRESS_XGBoost_N8_t6", "algo": "XGBoost", "regime": "STRESS", "horizon": 2, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "GE_ret_1d", "BDX_Becton_Dickinson_ret_20d", "HangSeng_HK_vol_20d", "SJM_JM_Smucker_ret_1d", "Michigan_Sentiment_ret_20d", "EOG_EOGResources_vol_20d"], "is_new": true}, {"model_id": "new_h2_STRESS_XGBoost_N8_t7", "algo": "XGBoost", "regime": "STRESS", "horizon": 2, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "CPB_CampbellSoup_zscore_60d", "CLX_Clorox_vol_20d", "Core_CPI_zscore_60d", "IYM_BasicMaterials_ret_20d", "EFFR_vol_20d", "IWM_SmallCap_vol_20d"], "is_new": true}, {"model_id": "new_h2_STRESS_XGBoost_N10_t0", "algo": "XGBoost", "regime": "STRESS", "horizon": 2, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "PAYX_Paychex_vol_20d", "TXN_vol_20d", "US1Y_Rate_ret_20d", "TM_Telephone_ret_1d", "EQIX_Equinix_ret_5d", "M_Macys_vol_20d", "AVB_AvalonBay_zscore_60d", "EWL_Switzerland_zscore_60d"], "is_new": true}, {"model_id": "new_h2_STRESS_XGBoost_N10_t1", "algo": "XGBoost", "regime": "STRESS", "horizon": 2, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWQ_France_ret_20d", "AMT_AmericanTower_ret_1d", "EXC_Exelon_ret_1d", "LOW_Lowes_ret_20d", "TGT_Target_zscore_60d", "SLB_Schlumberger_ret_1d", "3M_vol_20d", "VRP_ma5"], "is_new": true}, {"model_id": "new_h2_STRESS_XGBoost_N10_t2", "algo": "XGBoost", "regime": "STRESS", "horizon": 2, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "IBEX_Spain_ret_20d", "DE_Deere_ret_5d", "EQIX_Equinix_ret_5d", "XLF_Fin_vol_20d", "EWL_Switzerland_zscore_60d", "MO_AltriaMG_ret_1d", "IYM_BasicMaterials_ret_20d", "HUM_Humana_ret_5d"], "is_new": true}, {"model_id": "new_h2_STRESS_XGBoost_N10_t3", "algo": "XGBoost", "regime": "STRESS", "horizon": 2, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "IWM_SmallCap_vol_20d", "LMT_LockheedMartin_vol_20d", "US3M_Rate_vol_20d", "NEE_NextEra_ret_20d", "PAYX_Paychex_ret_20d", "ITT_ITTInc_ret_5d", "SCHW_Schwab_ret_5d", "WTI_Oil_FRED_zscore_60d"], "is_new": true}, {"model_id": "new_h2_STRESS_XGBoost_N10_t4", "algo": "XGBoost", "regime": "STRESS", "horizon": 2, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "PAYX_Paychex_vol_20d", "VRP_ma5", "T_ret_1d", "EWM_Malaysia_ret_1d", "AMGN_Amgen_ret_1d", "HangSeng_HK_vol_20d", "IYM_BasicMaterials_ret_20d", "PLD_Prologis_ret_5d"], "is_new": true}, {"model_id": "new_h2_STRESS_XGBoost_N10_t5", "algo": "XGBoost", "regime": "STRESS", "horizon": 2, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "IWM_SmallCap_vol_20d", "CPB_CampbellSoup_vol_20d", "CLX_Clorox_vol_20d", "MS_MorganStanley_ret_1d", "3M_ret_5d", "T_ret_1d", "US3M_Rate_zscore_60d", "LOW_Lowes_ret_20d"], "is_new": true}, {"model_id": "new_h2_STRESS_XGBoost_N10_t6", "algo": "XGBoost", "regime": "STRESS", "horizon": 2, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "LMT_LockheedMartin_ret_1d", "spx_momentum_3d", "ENB_EnbridgeInc_ret_1d", "EQR_Equity_ret_1d", "Core_CPI_zscore_60d", "EWG_Germany_vol_20d", "EWG_Germany_ret_20d", "MSTR_Bitcoin3_ret_20d"], "is_new": true}, {"model_id": "new_h2_STRESS_XGBoost_N10_t7", "algo": "XGBoost", "regime": "STRESS", "horizon": 2, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "SO_SouthernCo_ret_5d", "EWC_Canada_zscore_60d", "GE_ret_1d", "PFE_ret_1d", "Nikkei_Japan_zscore_60d", "Brent_Oil_FRED_ret_20d", "DE_Deere_ret_5d", "vix_mean_abs_ret_5d"], "is_new": true}, {"model_id": "new_h2_STRESS_XGBoost_N12_t0", "algo": "XGBoost", "regime": "STRESS", "horizon": 2, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "HD_ret_1d", "MS_MorganStanley_ret_1d", "CPB_CampbellSoup_zscore_60d", "Core_CPI_zscore_60d", "EWY_Korea_ret_20d", "US3Y_Rate_ret_5d", "XLB_Materials_zscore_60d", "heston_var_ev_h3", "Retail_Sales_zscore_60d", "US5Y_Rate_ret_5d"], "is_new": true}, {"model_id": "new_h2_STRESS_XGBoost_N12_t1", "algo": "XGBoost", "regime": "STRESS", "horizon": 2, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "heston_ev_h3", "IYM_BasicMaterials_ret_20d", "MS_MorganStanley_ret_5d", "AXP_Amex_vol_20d", "EWL_Switzerland_zscore_60d", "EWJ_Japan_vol_20d", "T10Y2Y_Spread_ret_5d", "SBUX_vol_20d", "TM_Telephone_vol_20d", "SPY_zscore_60d"], "is_new": true}, {"model_id": "new_h2_STRESS_XGBoost_N12_t2", "algo": "XGBoost", "regime": "STRESS", "horizon": 2, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "heston_var_ev_h5", "US30Y_Rate_ret_20d", "XLV_Health_zscore_60d", "SO_SouthernCo_ret_5d", "MS_MorganStanley_zscore_60d", "hmm_p_stress", "CPB_CampbellSoup_vol_20d", "TXN_vol_20d", "EXC_Exelon_zscore_60d", "EWQ_France_ret_20d"], "is_new": true}, {"model_id": "new_h2_STRESS_XGBoost_N12_t3", "algo": "XGBoost", "regime": "STRESS", "horizon": 2, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "BA_ret_1d", "US3M_Rate_zscore_60d", "AMD_ret_1d", "heston_var_ev_h7", "3M_vol_20d", "PPL_PPL_ret_1d", "EWQ_France_ret_20d", "SJM_JM_Smucker_ret_1d", "VOD_Vodafone_zscore_60d", "HD_ret_20d"], "is_new": true}, {"model_id": "new_h2_STRESS_XGBoost_N12_t4", "algo": "XGBoost", "regime": "STRESS", "horizon": 2, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "PG_ret_20d", "heston_var_ev_h5", "spx_momentum_3d", "EWY_Korea_zscore_60d", "IBEX_Spain_ret_20d", "GILD_Gilead_ret_20d", "CTAS_Cintas_vol_20d", "AXP_Amex_ret_20d", "PAYX_Paychex_zscore_60d", "Michigan_Sentiment_ret_20d"], "is_new": true}, {"model_id": "new_h2_STRESS_XGBoost_N12_t5", "algo": "XGBoost", "regime": "STRESS", "horizon": 2, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "MSTR_Bitcoin3_ret_5d", "heston_var_ev_h3", "JNJ_ret_1d", "HD_ret_20d", "spx_abs_ret_max_5d", "EWM_Malaysia_zscore_60d", "HangSeng_HK_ret_1d", "PAYX_Paychex_zscore_60d", "LMT_LockheedMartin_vol_20d", "US3M_Rate_zscore_60d"], "is_new": true}, {"model_id": "new_h2_STRESS_XGBoost_N12_t6", "algo": "XGBoost", "regime": "STRESS", "horizon": 2, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWY_Korea_ret_20d", "JNJ_ret_1d", "US6M_Rate_ret_20d", "hmm_p_stress", "SO_SouthernCo_ret_5d", "EWH_HongKong_ret_5d", "XLK_Tech_zscore_60d", "SPY_zscore_60d", "IYM_BasicMaterials_ret_20d", "XOM_ret_20d"], "is_new": true}, {"model_id": "new_h2_STRESS_XGBoost_N12_t7", "algo": "XGBoost", "regime": "STRESS", "horizon": 2, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "LUV_SouthwestAir_ret_5d", "SLB_Schlumberger_ret_1d", "INTC_ret_1d", "PAYX_Paychex_zscore_60d", "EWM_Malaysia_ret_1d", "SJM_JM_Smucker_ret_1d", "IYM_BasicMaterials_ret_20d", "NOC_Northrop_ret_20d", "vix_acceleration_1d", "US1Y_Rate_ret_20d"], "is_new": true}, {"model_id": "new_h2_STRESS_XGBoost_N15_t0", "algo": "XGBoost", "regime": "STRESS", "horizon": 2, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWL_Switzerland_vol_20d", "HangSeng_HK_vol_20d", "ASX_Australia_vol_20d", "SBUX_zscore_60d", "EWH_HongKong_ret_5d", "BDX_Becton_Dickinson_ret_20d", "PAYX_Paychex_ret_20d", "hmm_p_stress", "vix_mean_abs_ret_5d", "CPB_CampbellSoup_ret_5d", "PLD_Prologis_ret_5d", "MS_MorganStanley_ret_1d", "CPB_CampbellSoup_ret_20d"], "is_new": true}, {"model_id": "new_h2_STRESS_XGBoost_N15_t1", "algo": "XGBoost", "regime": "STRESS", "horizon": 2, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "heston_var_ev_h5", "XOM_ret_20d", "3M_ret_5d", "SBUX_zscore_60d", "SLB_Schlumberger_ret_5d", "TGT_Target_zscore_60d", "EWL_Switzerland_vol_20d", "MS_MorganStanley_ret_1d", "US7Y_Rate_ret_20d", "EFFR_vol_20d", "DOW_Price_zscore_60d", "EWA_Australia_ret_1d", "EWC_Canada_zscore_60d"], "is_new": true}, {"model_id": "new_h2_STRESS_XGBoost_N15_t2", "algo": "XGBoost", "regime": "STRESS", "horizon": 2, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWQ_France_ret_20d", "EWA_Australia_ret_1d", "NEE_NextEra_ret_20d", "US6M_Rate_ret_20d", "VOD_Vodafone_zscore_60d", "XLB_Materials_zscore_60d", "JNJ_ret_1d", "DE_Deere_ret_5d", "US1Y_Rate_ret_20d", "IYR_US_REIT2_zscore_60d", "AXP_Amex_ret_20d", "ES_Evergy_ret_1d", "EWM_Malaysia_ret_1d"], "is_new": true}, {"model_id": "new_h2_STRESS_XGBoost_N15_t3", "algo": "XGBoost", "regime": "STRESS", "horizon": 2, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWC_Canada_zscore_60d", "EFFR_vol_20d", "PFE_ret_1d", "EWQ_France_ret_20d", "INTC_ret_1d", "vix_mean_abs_ret_5d", "IYM_BasicMaterials_ret_20d", "Core_PCE_zscore_60d", "HD_ret_20d", "XOM_ret_1d", "PAYX_Paychex_zscore_60d", "EFFR_ret_1d", "FedFunds_zscore_60d"], "is_new": true}, {"model_id": "new_h2_STRESS_XGBoost_N15_t4", "algo": "XGBoost", "regime": "STRESS", "horizon": 2, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "DAX_Germany_zscore_60d", "EWS_Singapore_ret_5d", "GD_GeneralDynamics_zscore_60d", "SBUX_ret_5d", "SBUX_vol_20d", "Brent_Oil_FRED_ret_5d", "HD_zscore_60d", "NWL_Newell_ret_20d", "XLK_Tech_zscore_60d", "IYM_BasicMaterials_ret_20d", "ASX_Australia_ret_5d", "US7Y_Rate_ret_20d", "LOW_Lowes_ret_20d"], "is_new": true}, {"model_id": "new_h2_STRESS_XGBoost_N15_t5", "algo": "XGBoost", "regime": "STRESS", "horizon": 2, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "AVB_AvalonBay_zscore_60d", "DOW_Price_zscore_60d", "ASX_Australia_vol_20d", "NFCI_ret_5d", "Core_CPI_zscore_60d", "PPL_PPL_ret_1d", "EWY_Korea_zscore_60d", "Brent_Oil_FRED_ret_20d", "EWA_Australia_zscore_60d", "NVDA_vol_20d", "IBEX_Spain_ret_20d", "FedFunds_zscore_60d", "LOW_Lowes_ret_5d"], "is_new": true}, {"model_id": "new_h2_STRESS_XGBoost_N15_t6", "algo": "XGBoost", "regime": "STRESS", "horizon": 2, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "CTAS_Cintas_vol_20d", "EFFR_vol_20d", "CPB_CampbellSoup_zscore_60d", "US1Y_Rate_ret_20d", "US3M_Rate_vol_20d", "EWY_Korea_ret_20d", "spx_vol_5d", "VVIX_ret_20d", "CMCSA_ret_1d", "heston_ev_h3", "DIS_vol_20d", "Retail_Sales_zscore_60d", "AMD_ret_5d"], "is_new": true}, {"model_id": "new_h2_STRESS_XGBoost_N15_t7", "algo": "XGBoost", "regime": "STRESS", "horizon": 2, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "Michigan_Sentiment_ret_20d", "CPB_CampbellSoup_vol_20d", "EFFR_ret_1d", "EOG_EOGResources_ret_5d", "DOW_Price_zscore_60d", "TM_Telephone_vol_20d", "hmm_p_stress", "EWQ_France_ret_20d", "spx_abs_ret_max_5d", "gjr_condvar_h1", "AMZN_ret_5d", "LUV_SouthwestAir_ret_5d", "MSTR_Bitcoin3_ret_5d"], "is_new": true}, {"model_id": "new_h2_STRESS_XGBoost_N20_t0", "algo": "XGBoost", "regime": "STRESS", "horizon": 2, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWY_Korea_ret_20d", "vix_mean_abs_ret_5d", "3M_vol_20d", "EXC_Exelon_ret_1d", "MO_AltriaMG_ret_1d", "ITT_ITTInc_ret_5d", "JNJ_ret_1d", "US6M_Rate_ret_20d", "IYR_US_REIT2_zscore_60d", "BA_ret_1d", "DAX_Germany_zscore_60d", "NEE_NextEra_ret_20d", "EWJ_Japan_vol_20d", "Nikkei_Japan_vol_20d", "SJM_JM_Smucker_ret_1d", "SBUX_vol_20d", "TM_Telephone_ret_1d", "XLF_Fin_vol_20d"], "is_new": true}, {"model_id": "new_h2_STRESS_XGBoost_N20_t1", "algo": "XGBoost", "regime": "STRESS", "horizon": 2, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "heston_var_ev_h7", "DHR_ret_1d", "SBUX_vol_20d", "NEE_NextEra_ret_20d", "EWA_Australia_zscore_60d", "US5Y_Rate_ret_5d", "WTI_Oil_FRED_zscore_60d", "BLK_BlackRock_zscore_60d", "FedFunds_zscore_60d", "NFCI_ret_5d", "ASX_Australia_vol_20d", "AORD_AUS_zscore_60d", "T10Y2Y_Spread_ret_5d", "EQIX_Equinix_ret_5d", "EWS_Singapore_ret_5d", "M_Macys_vol_20d", "US3M_Rate_vol_20d", "MSTR_Bitcoin3_ret_1d"], "is_new": true}, {"model_id": "new_h2_STRESS_XGBoost_N20_t2", "algo": "XGBoost", "regime": "STRESS", "horizon": 2, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "ES_Evergy_ret_1d", "DOW_Price_zscore_60d", "BLK_BlackRock_zscore_60d", "US3M_Rate_vol_20d", "PAYX_Paychex_ret_20d", "MRK_Merck_zscore_60d", "Industrial_Production_zscore_60d", "AMT_AmericanTower_ret_1d", "M_Macys_vol_20d", "CPB_CampbellSoup_vol_20d", "PFE_ret_1d", "XLF_Fin_vol_20d", "ASX_Australia_vol_20d", "Nikkei_Japan_zscore_60d", "heston_var_ev_h5", "PG_ret_20d", "AMZN_ret_5d", "BTI_BritishAmerican_ret_5d"], "is_new": true}, {"model_id": "new_h2_STRESS_XGBoost_N20_t3", "algo": "XGBoost", "regime": "STRESS", "horizon": 2, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "BDX_Becton_Dickinson_ret_20d", "EWH_HongKong_ret_5d", "US7Y_Rate_ret_20d", "EWY_Korea_zscore_60d", "IBEX_Spain_ret_20d", "XLK_Tech_zscore_60d", "ASX_Australia_ret_5d", "PFE_ret_1d", "ORCL_vol_20d", "WTI_Oil_FRED_zscore_60d", "QQQ_vol_20d", "EMR_Emerson_ret_20d", "Core_CPI_zscore_60d", "MSTR_Bitcoin3_ret_20d", "EQR_Equity_ret_1d", "Brent_Oil_FRED_ret_20d", "US1Y_Rate_ret_20d", "hmm_p_stress"], "is_new": true}, {"model_id": "new_h2_STRESS_XGBoost_N20_t4", "algo": "XGBoost", "regime": "STRESS", "horizon": 2, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "VVIX_ret_20d", "EFFR_vol_20d", "US5Y_Rate_ret_5d", "MSTR_Bitcoin3_ret_1d", "hmm_p_stress", "XLK_Tech_zscore_60d", "DIS_vol_20d", "CTAS_Cintas_vol_20d", "3M_ret_5d", "Industrial_Production_zscore_60d", "EWM_Malaysia_ret_1d", "AMZN_ret_5d", "Retail_Sales_zscore_60d", "CI_Cigna_vol_20d", "EOG_EOGResources_vol_20d", "NOC_Northrop_ret_20d", "AVB_AvalonBay_zscore_60d", "BDX_Becton_Dickinson_ret_20d"], "is_new": true}, {"model_id": "new_h2_STRESS_XGBoost_N20_t5", "algo": "XGBoost", "regime": "STRESS", "horizon": 2, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "spx_abs_ret_max_5d", "PG_ret_20d", "PAYX_Paychex_ret_20d", "HD_ret_1d", "EQR_Equity_ret_1d", "PCAR_PaccarInc_ret_5d", "BA_ret_1d", "INTC_ret_1d", "HD_ret_5d", "XLY_Disc_vol_20d", "US30Y_Rate_ret_20d", "ITT_ITTInc_ret_5d", "SBUX_zscore_60d", "TM_Telephone_vol_20d", "TM_Telephone_ret_1d", "ENB_EnbridgeInc_ret_1d", "T_ret_1d", "DIS_vol_20d"], "is_new": true}, {"model_id": "new_h2_STRESS_XGBoost_N20_t6", "algo": "XGBoost", "regime": "STRESS", "horizon": 2, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EFFR_vol_20d", "spx_vol_5d", "SPY_zscore_60d", "AMZN_ret_5d", "CCI_CrownCastle_vol_20d", "ENB_EnbridgeInc_ret_1d", "NVDA_vol_20d", "US5Y_Rate_ret_5d", "US1Y_Rate_ret_20d", "BA_ret_1d", "HD_zscore_60d", "EXC_Exelon_zscore_60d", "NOC_Northrop_ret_20d", "LUV_SouthwestAir_ret_5d", "SJM_JM_Smucker_ret_5d", "US6M_Rate_ret_20d", "ITT_ITTInc_ret_5d", "DHR_ret_1d"], "is_new": true}, {"model_id": "new_h2_STRESS_XGBoost_N20_t7", "algo": "XGBoost", "regime": "STRESS", "horizon": 2, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "AMD_ret_5d", "US3M_Rate_zscore_60d", "NOC_Northrop_ret_20d", "AMGN_Amgen_ret_1d", "heston_var_ev_h7", "Nikkei_Japan_vol_20d", "FedFunds_zscore_60d", "ORCL_zscore_60d", "EFFR_ret_1d", "XOM_ret_20d", "EXC_Exelon_ret_1d", "JNJ_ret_1d", "EQIX_Equinix_ret_5d", "LOW_Lowes_ret_5d", "PG_ret_20d", "EWL_Switzerland_vol_20d", "PAYX_Paychex_zscore_60d", "T_ret_1d"], "is_new": true}, {"model_id": "new_h2_STRESS_XGBoost_N25_t0", "algo": "XGBoost", "regime": "STRESS", "horizon": 2, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "T_ret_1d", "TM_Telephone_ret_1d", "EXC_Exelon_zscore_60d", "AMD_ret_5d", "PAYX_Paychex_ret_20d", "3M_ret_5d", "XLF_Fin_vol_20d", "GE_ret_1d", "NOC_Northrop_ret_20d", "INTC_ret_5d", "Retail_Sales_zscore_60d", "PLD_Prologis_ret_5d", "PFE_ret_1d", "BA_ret_1d", "ORCL_vol_20d", "CLX_Clorox_vol_20d", "DHR_vol_20d", "SO_SouthernCo_ret_5d", "SJM_JM_Smucker_ret_5d", "PCAR_PaccarInc_ret_5d", "EFFR_ret_1d", "TXN_vol_20d", "HangSeng_HK_vol_20d"], "is_new": true}, {"model_id": "new_h2_STRESS_XGBoost_N25_t1", "algo": "XGBoost", "regime": "STRESS", "horizon": 2, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "SO_SouthernCo_ret_5d", "DIS_vol_20d", "M_Macys_vol_20d", "Brent_Oil_FRED_ret_20d", "heston_var_ev_h5", "MSTR_Bitcoin3_ret_1d", "EXC_Exelon_ret_1d", "Core_PCE_zscore_60d", "ORCL_zscore_60d", "AMZN_ret_5d", "3M_ret_5d", "JNJ_ret_1d", "EWC_Canada_zscore_60d", "SCHW_Schwab_ret_5d", "Brent_Oil_FRED_ret_5d", "SPY_zscore_60d", "NWL_Newell_ret_20d", "ASX_Australia_ret_5d", "IYR_US_REIT2_zscore_60d", "MS_MorganStanley_ret_1d", "MS_MorganStanley_zscore_60d", "TM_Telephone_vol_20d", "vix_mean_abs_ret_5d"], "is_new": true}, {"model_id": "new_h2_STRESS_XGBoost_N25_t2", "algo": "XGBoost", "regime": "STRESS", "horizon": 2, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "MSTR_Bitcoin3_ret_1d", "HangSeng_HK_vol_20d", "PPL_PPL_ret_1d", "US5Y_Rate_ret_5d", "MO_AltriaMG_ret_1d", "heston_ev_h3", "BTI_BritishAmerican_ret_20d", "DHR_vol_20d", "PLD_Prologis_ret_5d", "US7Y_Rate_ret_20d", "TED_Spread_vol_20d", "EWA_Australia_ret_1d", "DAX_Germany_vol_20d", "EOG_EOGResources_vol_20d", "FedFunds_zscore_60d", "DE_Deere_vol_20d", "US1Y_Rate_ret_5d", "LOW_Lowes_ret_5d", "EWQ_France_ret_20d", "MRK_Merck_zscore_60d", "GE_ret_1d", "CI_Cigna_vol_20d", "M_Macys_vol_20d"], "is_new": true}, {"model_id": "new_h2_STRESS_XGBoost_N25_t3", "algo": "XGBoost", "regime": "STRESS", "horizon": 2, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "US1Y_Rate_ret_5d", "AMT_AmericanTower_ret_1d", "T10Y2Y_Spread_ret_5d", "ORCL_zscore_60d", "3M_ret_5d", "Core_PCE_zscore_60d", "PPL_PPL_ret_1d", "spx_abs_ret_max_5d", "SO_SouthernCo_ret_5d", "M_Macys_vol_20d", "SBUX_vol_20d", "CPB_CampbellSoup_vol_20d", "LUV_SouthwestAir_ret_5d", "EOG_EOGResources_ret_5d", "AVB_AvalonBay_zscore_60d", "Retail_Sales_zscore_60d", "PLD_Prologis_ret_5d", "XOM_ret_20d", "BDX_Becton_Dickinson_ret_20d", "heston_var_ev_h7", "HD_ret_5d", "WTI_Oil_FRED_zscore_60d", "HD_ret_20d"], "is_new": true}, {"model_id": "new_h2_STRESS_XGBoost_N25_t4", "algo": "XGBoost", "regime": "STRESS", "horizon": 2, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "PAYX_Paychex_ret_20d", "VVIX_ret_20d", "MSTR_Bitcoin3_ret_20d", "PAYX_Paychex_zscore_60d", "DOW_Price_zscore_60d", "heston_var_ev_h7", "HUM_Humana_ret_5d", "NFCI_ret_5d", "CI_Cigna_vol_20d", "BA_ret_1d", "IWM_SmallCap_vol_20d", "BTI_BritishAmerican_ret_5d", "CMCSA_ret_1d", "ASX_Australia_ret_5d", "US3Y_Rate_ret_5d", "3M_ret_5d", "heston_ev_h3", "PPL_PPL_ret_1d", "XLF_Fin_vol_20d", "CPB_CampbellSoup_zscore_60d", "SJM_JM_Smucker_ret_1d", "US1Y_Rate_ret_5d", "LOW_Lowes_ret_5d"], "is_new": true}, {"model_id": "new_h2_STRESS_XGBoost_N25_t5", "algo": "XGBoost", "regime": "STRESS", "horizon": 2, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "AXP_Amex_vol_20d", "DOW_Price_zscore_60d", "LMT_LockheedMartin_ret_1d", "HD_ret_5d", "US3M_Rate_vol_20d", "QQQ_vol_20d", "EMR_Emerson_ret_20d", "XOM_ret_20d", "EWM_Malaysia_vol_20d", "XLY_Disc_vol_20d", "EOG_EOGResources_vol_20d", "ASX_Australia_ret_5d", "CCI_CrownCastle_vol_20d", "AMD_ret_5d", "gjr_condvar_h1", "PPL_PPL_ret_1d", "LUV_SouthwestAir_ret_5d", "XLB_Materials_zscore_60d", "hmm_p_stress", "GD_GeneralDynamics_zscore_60d", "PFE_ret_1d", "IYR_US_REIT2_zscore_60d", "heston_var_ev_h3"], "is_new": true}, {"model_id": "new_h2_STRESS_XGBoost_N25_t6", "algo": "XGBoost", "regime": "STRESS", "horizon": 2, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "IBEX_Spain_ret_20d", "EXC_Exelon_ret_1d", "ENB_EnbridgeInc_ret_1d", "LLY_zscore_60d", "DOW_Price_zscore_60d", "MSTR_Bitcoin3_ret_5d", "XLV_Health_zscore_60d", "EXC_Exelon_zscore_60d", "spx_vol_5d", "BDX_Becton_Dickinson_ret_20d", "HD_ret_20d", "CI_Cigna_vol_20d", "Michigan_Sentiment_ret_20d", "gjr_condvar_h1", "CTAS_Cintas_vol_20d", "LMT_LockheedMartin_ret_1d", "EWJ_Japan_vol_20d", "ORCL_zscore_60d", "HD_zscore_60d", "AVB_AvalonBay_zscore_60d", "LUV_SouthwestAir_ret_5d", "CMCSA_ret_1d", "EWA_Australia_ret_1d"], "is_new": true}, {"model_id": "new_h2_STRESS_XGBoost_N25_t7", "algo": "XGBoost", "regime": "STRESS", "horizon": 2, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "NWL_Newell_ret_20d", "MS_MorganStanley_zscore_60d", "US3Y_Rate_ret_5d", "XLF_Fin_vol_20d", "PAYX_Paychex_vol_20d", "MSTR_Bitcoin3_ret_20d", "TGT_Target_zscore_60d", "Core_CPI_zscore_60d", "MSTR_Bitcoin3_ret_1d", "3M_vol_20d", "TED_Spread_zscore_60d", "INTC_ret_5d", "JNJ_ret_1d", "SJM_JM_Smucker_ret_1d", "CLX_Clorox_vol_20d", "PFE_ret_1d", "HangSeng_HK_ret_5d", "WTI_Oil_FRED_zscore_60d", "AXP_Amex_ret_20d", "LOW_Lowes_ret_5d", "T10Y2Y_Spread_ret_5d", "HangSeng_HK_ret_1d", "MS_MorganStanley_ret_5d"], "is_new": true}, {"model_id": "new_h2_STRESS_XGBoost_N30_t0", "algo": "XGBoost", "regime": "STRESS", "horizon": 2, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "VOD_Vodafone_zscore_60d", "XLK_Tech_zscore_60d", "ASX_Australia_ret_5d", "SO_SouthernCo_ret_5d", "BDX_Becton_Dickinson_ret_20d", "EWJ_Japan_vol_20d", "Industrial_Production_zscore_60d", "SJM_JM_Smucker_ret_5d", "M_Macys_vol_20d", "EOG_EOGResources_ret_5d", "HD_ret_20d", "AXP_Amex_ret_20d", "LOW_Lowes_ret_20d", "PAYX_Paychex_ret_20d", "EXC_Exelon_ret_1d", "FedFunds_zscore_60d", "IYM_BasicMaterials_ret_20d", "AMZN_ret_5d", "US3M_Rate_vol_20d", "US5Y_Rate_ret_5d", "US1Y_Rate_ret_5d", "ORCL_vol_20d", "EWH_HongKong_ret_5d", "EQR_Equity_ret_1d", "EXC_Exelon_zscore_60d", "MRK_Merck_zscore_60d", "EWS_Singapore_ret_5d", "INTC_ret_5d"], "is_new": true}, {"model_id": "new_h2_STRESS_XGBoost_N30_t1", "algo": "XGBoost", "regime": "STRESS", "horizon": 2, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "Core_CPI_zscore_60d", "CMCSA_ret_1d", "BDX_Becton_Dickinson_ret_20d", "SPY_zscore_60d", "MRK_Merck_zscore_60d", "Brent_Oil_FRED_ret_20d", "TGT_Target_zscore_60d", "EWY_Korea_zscore_60d", "NFCI_ret_5d", "NWL_Newell_ret_20d", "SLB_Schlumberger_ret_5d", "XLK_Tech_zscore_60d", "US1Y_Rate_ret_20d", "ITT_ITTInc_ret_5d", "VVIX_ret_20d", "EWL_Switzerland_zscore_60d", "ENB_EnbridgeInc_ret_1d", "Brent_Oil_FRED_ret_5d", "EWS_Singapore_ret_5d", "LOW_Lowes_ret_20d", "US5Y_Rate_ret_5d", "heston_var_ev_h3", "Core_PCE_zscore_60d", "SJM_JM_Smucker_ret_5d", "EWH_HongKong_ret_5d", "BA_ret_1d", "PCAR_PaccarInc_ret_5d", "EWC_Canada_zscore_60d"], "is_new": true}, {"model_id": "new_h2_STRESS_XGBoost_N30_t2", "algo": "XGBoost", "regime": "STRESS", "horizon": 2, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EFFR_ret_1d", "DIS_vol_20d", "spx_momentum_3d", "NVDA_vol_20d", "FedFunds_zscore_60d", "vix_mean_abs_ret_5d", "LLY_zscore_60d", "XLV_Health_zscore_60d", "SCHW_Schwab_ret_5d", "DE_Deere_vol_20d", "AMD_ret_1d", "IYR_US_REIT2_zscore_60d", "IYM_BasicMaterials_ret_20d", "HUM_Humana_ret_5d", "TXN_vol_20d", "EMR_Emerson_ret_20d", "US5Y_Rate_ret_5d", "SPY_zscore_60d", "SBUX_ret_5d", "T_ret_1d", "CPB_CampbellSoup_vol_20d", "DHR_vol_20d", "BTI_BritishAmerican_ret_20d", "HD_ret_20d", "PPL_PPL_ret_1d", "EXC_Exelon_ret_1d", "TM_Telephone_vol_20d", "HD_ret_1d"], "is_new": true}, {"model_id": "new_h2_STRESS_XGBoost_N30_t3", "algo": "XGBoost", "regime": "STRESS", "horizon": 2, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "HUM_Humana_ret_5d", "EOG_EOGResources_vol_20d", "US7Y_Rate_ret_20d", "EWY_Korea_ret_20d", "PCAR_PaccarInc_ret_5d", "CPB_CampbellSoup_vol_20d", "SBUX_ret_5d", "T_ret_1d", "EWY_Korea_zscore_60d", "QQQ_vol_20d", "EQR_Equity_ret_1d", "CCI_CrownCastle_vol_20d", "FedFunds_zscore_60d", "Nikkei_Japan_zscore_60d", "XOM_ret_20d", "PFE_ret_1d", "gjr_condvar_h1", "SJM_JM_Smucker_ret_1d", "Industrial_Production_zscore_60d", "TM_Telephone_ret_1d", "AXP_Amex_vol_20d", "GE_ret_1d", "LLY_zscore_60d", "DHR_vol_20d", "XLF_Fin_vol_20d", "NWL_Newell_ret_20d", "PAYX_Paychex_vol_20d", "Nikkei_Japan_vol_20d"], "is_new": true}, {"model_id": "new_h2_STRESS_XGBoost_N30_t4", "algo": "XGBoost", "regime": "STRESS", "horizon": 2, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "IYM_BasicMaterials_ret_20d", "HUM_Humana_ret_5d", "HD_ret_1d", "vix_mean_abs_ret_5d", "SO_SouthernCo_ret_5d", "EXC_Exelon_zscore_60d", "PAYX_Paychex_zscore_60d", "vix_acceleration_1d", "BTI_BritishAmerican_ret_5d", "EWQ_France_ret_20d", "spx_vol_5d", "3M_vol_20d", "Core_CPI_zscore_60d", "XLK_Tech_zscore_60d", "DE_Deere_vol_20d", "US6M_Rate_ret_20d", "EWJ_Japan_vol_20d", "NOC_Northrop_ret_20d", "3M_ret_5d", "spx_momentum_3d", "ORCL_vol_20d", "EQR_Equity_ret_1d", "PCAR_PaccarInc_ret_5d", "IYR_US_REIT2_zscore_60d", "BTI_BritishAmerican_ret_20d", "EWH_HongKong_ret_5d", "EWY_Korea_zscore_60d", "MSTR_Bitcoin3_ret_1d"], "is_new": true}, {"model_id": "new_h2_STRESS_XGBoost_N30_t5", "algo": "XGBoost", "regime": "STRESS", "horizon": 2, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "HD_ret_1d", "TED_Spread_zscore_60d", "HD_ret_5d", "HUM_Humana_ret_5d", "Retail_Sales_zscore_60d", "XLB_Materials_zscore_60d", "EWS_Singapore_ret_5d", "NWL_Newell_ret_20d", "LMT_LockheedMartin_ret_1d", "NEE_NextEra_ret_20d", "Core_PCE_zscore_60d", "BTI_BritishAmerican_ret_5d", "LOW_Lowes_ret_5d", "heston_ev_h3", "US1Y_Rate_ret_20d", "M_Macys_vol_20d", "DE_Deere_vol_20d", "DE_Deere_ret_5d", "EWQ_France_zscore_60d", "BLK_BlackRock_zscore_60d", "PLD_Prologis_ret_5d", "AORD_AUS_zscore_60d", "NVDA_vol_20d", "IYR_US_REIT2_zscore_60d", "EWY_Korea_zscore_60d", "SPY_zscore_60d", "VOD_Vodafone_zscore_60d", "INTC_ret_1d"], "is_new": true}, {"model_id": "new_h2_STRESS_XGBoost_N30_t6", "algo": "XGBoost", "regime": "STRESS", "horizon": 2, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "DE_Deere_ret_5d", "vix_mean_abs_ret_5d", "PAYX_Paychex_zscore_60d", "heston_ev_h3", "HangSeng_HK_ret_1d", "EWQ_France_zscore_60d", "EWY_Korea_ret_20d", "BA_ret_1d", "EWJ_Japan_vol_20d", "ORCL_zscore_60d", "EQIX_Equinix_ret_5d", "SBUX_zscore_60d", "BTI_BritishAmerican_ret_5d", "TM_Telephone_ret_1d", "NFCI_ret_5d", "EMR_Emerson_ret_20d", "HD_ret_20d", "spx_abs_ret_max_5d", "PCAR_PaccarInc_ret_5d", "LMT_LockheedMartin_vol_20d", "NVDA_vol_20d", "EWA_Australia_ret_1d", "BTI_BritishAmerican_ret_20d", "CPB_CampbellSoup_vol_20d", "vix_acceleration_1d", "Brent_Oil_FRED_ret_5d", "EWH_HongKong_ret_5d", "TXN_vol_20d"], "is_new": true}, {"model_id": "new_h2_STRESS_XGBoost_N30_t7", "algo": "XGBoost", "regime": "STRESS", "horizon": 2, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "US1Y_Rate_ret_5d", "SBUX_vol_20d", "TM_Telephone_vol_20d", "AXP_Amex_vol_20d", "MS_MorganStanley_zscore_60d", "NOC_Northrop_ret_20d", "Brent_Oil_FRED_ret_5d", "INTC_ret_5d", "HangSeng_HK_ret_1d", "LLY_zscore_60d", "XOM_ret_1d", "SLB_Schlumberger_ret_5d", "IYM_BasicMaterials_ret_20d", "SO_SouthernCo_ret_5d", "M_Macys_vol_20d", "MSTR_Bitcoin3_ret_1d", "EWM_Malaysia_zscore_60d", "LMT_LockheedMartin_vol_20d", "Nikkei_Japan_zscore_60d", "IWM_SmallCap_vol_20d", "HUM_Humana_ret_5d", "CCI_CrownCastle_vol_20d", "T10Y2Y_Spread_ret_5d", "AORD_AUS_zscore_60d", "ENB_EnbridgeInc_ret_1d", "US1Y_Rate_ret_20d", "US7Y_Rate_ret_20d", "EQIX_Equinix_ret_5d"], "is_new": true}, {"model_id": "new_h2_STRESS_LightGBM_N5_t0", "algo": "LightGBM", "regime": "STRESS", "horizon": 2, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "spx_vol_5d", "T10Y2Y_Spread_ret_5d", "TM_Telephone_vol_20d"], "is_new": true}, {"model_id": "new_h2_STRESS_LightGBM_N5_t1", "algo": "LightGBM", "regime": "STRESS", "horizon": 2, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "XLV_Health_zscore_60d", "GE_ret_1d", "MS_MorganStanley_ret_1d"], "is_new": true}, {"model_id": "new_h2_STRESS_LightGBM_N5_t2", "algo": "LightGBM", "regime": "STRESS", "horizon": 2, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "TGT_Target_zscore_60d", "HD_ret_5d", "PAYX_Paychex_vol_20d"], "is_new": true}, {"model_id": "new_h2_STRESS_LightGBM_N5_t3", "algo": "LightGBM", "regime": "STRESS", "horizon": 2, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWS_Singapore_ret_5d", "CCI_CrownCastle_vol_20d", "vix_mean_abs_ret_5d"], "is_new": true}, {"model_id": "new_h2_STRESS_LightGBM_N5_t4", "algo": "LightGBM", "regime": "STRESS", "horizon": 2, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "ENB_EnbridgeInc_ret_1d", "EWH_HongKong_ret_5d", "EFFR_vol_20d"], "is_new": true}, {"model_id": "new_h2_STRESS_LightGBM_N5_t5", "algo": "LightGBM", "regime": "STRESS", "horizon": 2, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "T10Y2Y_Spread_ret_5d", "ORCL_vol_20d", "HUM_Humana_ret_5d"], "is_new": true}, {"model_id": "new_h2_STRESS_LightGBM_N5_t6", "algo": "LightGBM", "regime": "STRESS", "horizon": 2, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "DE_Deere_vol_20d", "Nikkei_Japan_vol_20d", "TM_Telephone_vol_20d"], "is_new": true}, {"model_id": "new_h2_STRESS_LightGBM_N5_t7", "algo": "LightGBM", "regime": "STRESS", "horizon": 2, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "DHR_vol_20d", "TED_Spread_zscore_60d", "PG_ret_20d"], "is_new": true}, {"model_id": "new_h2_STRESS_LightGBM_N8_t0", "algo": "LightGBM", "regime": "STRESS", "horizon": 2, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "DOW_Price_zscore_60d", "3M_vol_20d", "SLB_Schlumberger_ret_1d", "SBUX_ret_5d", "PAYX_Paychex_vol_20d", "US30Y_Rate_ret_20d"], "is_new": true}, {"model_id": "new_h2_STRESS_LightGBM_N8_t1", "algo": "LightGBM", "regime": "STRESS", "horizon": 2, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "3M_vol_20d", "NEE_NextEra_ret_20d", "EQR_Equity_ret_1d", "vix_acceleration_1d", "TM_Telephone_ret_1d", "EWM_Malaysia_vol_20d"], "is_new": true}, {"model_id": "new_h2_STRESS_LightGBM_N8_t2", "algo": "LightGBM", "regime": "STRESS", "horizon": 2, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "Retail_Sales_zscore_60d", "EWS_Singapore_ret_5d", "JNJ_ret_1d", "NWL_Newell_ret_20d", "EOG_EOGResources_vol_20d", "EWA_Australia_zscore_60d"], "is_new": true}, {"model_id": "new_h2_STRESS_LightGBM_N8_t3", "algo": "LightGBM", "regime": "STRESS", "horizon": 2, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "heston_var_ev_h5", "ASX_Australia_vol_20d", "BDX_Becton_Dickinson_ret_20d", "EXC_Exelon_ret_1d", "IYM_BasicMaterials_ret_20d", "EWJ_Japan_vol_20d"], "is_new": true}, {"model_id": "new_h2_STRESS_LightGBM_N8_t4", "algo": "LightGBM", "regime": "STRESS", "horizon": 2, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "BLK_BlackRock_zscore_60d", "DAX_Germany_zscore_60d", "PG_ret_20d", "EWL_Switzerland_zscore_60d", "heston_ev_h3", "vix_mean_abs_ret_5d"], "is_new": true}, {"model_id": "new_h2_STRESS_LightGBM_N8_t5", "algo": "LightGBM", "regime": "STRESS", "horizon": 2, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "AMT_AmericanTower_ret_1d", "IBEX_Spain_ret_20d", "EMR_Emerson_ret_20d", "EXC_Exelon_ret_1d", "AVB_AvalonBay_zscore_60d", "EFFR_ret_1d"], "is_new": true}, {"model_id": "new_h2_STRESS_LightGBM_N8_t6", "algo": "LightGBM", "regime": "STRESS", "horizon": 2, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "NFCI_ret_5d", "T10Y2Y_Spread_ret_5d", "XLB_Materials_zscore_60d", "EWM_Malaysia_ret_1d", "US6M_Rate_ret_20d", "PAYX_Paychex_zscore_60d"], "is_new": true}, {"model_id": "new_h2_STRESS_LightGBM_N8_t7", "algo": "LightGBM", "regime": "STRESS", "horizon": 2, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "CPB_CampbellSoup_vol_20d", "NOC_Northrop_ret_20d", "Retail_Sales_zscore_60d", "US1Y_Rate_ret_5d", "TM_Telephone_ret_1d", "BTI_BritishAmerican_ret_5d"], "is_new": true}, {"model_id": "new_h2_STRESS_LightGBM_N10_t0", "algo": "LightGBM", "regime": "STRESS", "horizon": 2, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "heston_var_ev_h7", "PFE_ret_1d", "EWM_Malaysia_zscore_60d", "TED_Spread_zscore_60d", "MS_MorganStanley_ret_5d", "JNJ_ret_1d", "MSTR_Bitcoin3_ret_20d", "XLF_Fin_vol_20d"], "is_new": true}, {"model_id": "new_h2_STRESS_LightGBM_N10_t1", "algo": "LightGBM", "regime": "STRESS", "horizon": 2, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "vix_mean_abs_ret_5d", "Nikkei_Japan_zscore_60d", "SJM_JM_Smucker_ret_1d", "EWM_Malaysia_vol_20d", "EWH_HongKong_ret_5d", "ES_Evergy_ret_1d", "EQIX_Equinix_ret_5d", "XOM_ret_1d"], "is_new": true}, {"model_id": "new_h2_STRESS_LightGBM_N10_t2", "algo": "LightGBM", "regime": "STRESS", "horizon": 2, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "LMT_LockheedMartin_vol_20d", "TGT_Target_zscore_60d", "PLD_Prologis_ret_5d", "ORCL_zscore_60d", "AORD_AUS_zscore_60d", "AMD_ret_1d", "spx_momentum_3d", "NWL_Newell_ret_20d"], "is_new": true}, {"model_id": "new_h2_STRESS_LightGBM_N10_t3", "algo": "LightGBM", "regime": "STRESS", "horizon": 2, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "US7Y_Rate_ret_20d", "VRP_ma5", "HD_ret_1d", "EFFR_vol_20d", "NVDA_vol_20d", "GILD_Gilead_ret_20d", "ASX_Australia_ret_5d", "SBUX_vol_20d"], "is_new": true}, {"model_id": "new_h2_STRESS_LightGBM_N10_t4", "algo": "LightGBM", "regime": "STRESS", "horizon": 2, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "BLK_BlackRock_zscore_60d", "CI_Cigna_vol_20d", "DAX_Germany_zscore_60d", "AXP_Amex_vol_20d", "EWM_Malaysia_ret_1d", "EWS_Singapore_ret_5d", "US5Y_Rate_ret_5d", "SLB_Schlumberger_ret_5d"], "is_new": true}, {"model_id": "new_h2_STRESS_LightGBM_N10_t5", "algo": "LightGBM", "regime": "STRESS", "horizon": 2, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "CMCSA_ret_1d", "MSTR_Bitcoin3_ret_1d", "DHR_vol_20d", "TXN_vol_20d", "EWM_Malaysia_ret_1d", "EWY_Korea_ret_20d", "LOW_Lowes_ret_20d", "NWL_Newell_ret_20d"], "is_new": true}, {"model_id": "new_h2_STRESS_LightGBM_N10_t6", "algo": "LightGBM", "regime": "STRESS", "horizon": 2, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "ASX_Australia_vol_20d", "TM_Telephone_vol_20d", "AMGN_Amgen_ret_1d", "Core_CPI_zscore_60d", "LUV_SouthwestAir_ret_5d", "EWL_Switzerland_vol_20d", "EWG_Germany_vol_20d", "PFE_ret_1d"], "is_new": true}, {"model_id": "new_h2_STRESS_LightGBM_N10_t7", "algo": "LightGBM", "regime": "STRESS", "horizon": 2, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "PFE_ret_1d", "AMT_AmericanTower_ret_1d", "DE_Deere_ret_5d", "heston_ev_h3", "spx_abs_ret_max_5d", "PAYX_Paychex_zscore_60d", "SBUX_zscore_60d", "ENB_EnbridgeInc_ret_1d"], "is_new": true}, {"model_id": "new_h2_STRESS_LightGBM_N12_t0", "algo": "LightGBM", "regime": "STRESS", "horizon": 2, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "MSTR_Bitcoin3_ret_20d", "LUV_SouthwestAir_ret_5d", "XLF_Fin_vol_20d", "Nikkei_Japan_zscore_60d", "spx_momentum_3d", "XLY_Disc_vol_20d", "MO_AltriaMG_ret_1d", "PG_ret_20d", "US6M_Rate_ret_20d", "vix_acceleration_1d"], "is_new": true}, {"model_id": "new_h2_STRESS_LightGBM_N12_t1", "algo": "LightGBM", "regime": "STRESS", "horizon": 2, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "US3M_Rate_zscore_60d", "JNJ_ret_1d", "XLK_Tech_zscore_60d", "MO_AltriaMG_ret_1d", "AMZN_ret_5d", "ITT_ITTInc_ret_5d", "hmm_p_stress", "EWQ_France_zscore_60d", "gjr_condvar_h1", "MS_MorganStanley_ret_1d"], "is_new": true}, {"model_id": "new_h2_STRESS_LightGBM_N12_t2", "algo": "LightGBM", "regime": "STRESS", "horizon": 2, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "HangSeng_HK_ret_1d", "SO_SouthernCo_ret_5d", "LUV_SouthwestAir_ret_5d", "HD_ret_5d", "Brent_Oil_FRED_ret_5d", "US30Y_Rate_ret_20d", "EWM_Malaysia_ret_1d", "XOM_ret_1d", "LLY_zscore_60d", "EMR_Emerson_ret_20d"], "is_new": true}, {"model_id": "new_h2_STRESS_LightGBM_N12_t3", "algo": "LightGBM", "regime": "STRESS", "horizon": 2, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "DHR_vol_20d", "BLK_BlackRock_zscore_60d", "PCAR_PaccarInc_ret_5d", "vix_mean_abs_ret_5d", "TED_Spread_zscore_60d", "heston_var_ev_h7", "SJM_JM_Smucker_ret_5d", "EMR_Emerson_ret_20d", "Brent_Oil_FRED_ret_5d", "MS_MorganStanley_ret_5d"], "is_new": true}, {"model_id": "new_h2_STRESS_LightGBM_N12_t4", "algo": "LightGBM", "regime": "STRESS", "horizon": 2, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "XLK_Tech_zscore_60d", "gjr_condvar_h1", "ES_Evergy_ret_1d", "CMCSA_ret_1d", "HangSeng_HK_ret_1d", "MRK_Merck_zscore_60d", "SBUX_zscore_60d", "HUM_Humana_ret_5d", "INTC_ret_5d", "DOW_Price_zscore_60d"], "is_new": true}, {"model_id": "new_h2_STRESS_LightGBM_N12_t5", "algo": "LightGBM", "regime": "STRESS", "horizon": 2, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "QQQ_vol_20d", "gjr_condvar_h1", "MSTR_Bitcoin3_ret_5d", "XOM_ret_1d", "FedFunds_zscore_60d", "PG_ret_20d", "EWG_Germany_ret_20d", "AMZN_ret_5d", "SO_SouthernCo_ret_5d", "T_ret_1d"], "is_new": true}, {"model_id": "new_h2_STRESS_LightGBM_N12_t6", "algo": "LightGBM", "regime": "STRESS", "horizon": 2, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "LOW_Lowes_ret_5d", "EWQ_France_zscore_60d", "EOG_EOGResources_vol_20d", "CMCSA_ret_1d", "EOG_EOGResources_ret_5d", "HD_ret_20d", "HD_ret_5d", "AMD_ret_1d", "US7Y_Rate_ret_20d", "EWG_Germany_ret_20d"], "is_new": true}, {"model_id": "new_h2_STRESS_LightGBM_N12_t7", "algo": "LightGBM", "regime": "STRESS", "horizon": 2, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "US5Y_Rate_ret_5d", "EWY_Korea_zscore_60d", "CMCSA_ret_1d", "EQIX_Equinix_ret_5d", "CLX_Clorox_vol_20d", "US1Y_Rate_ret_20d", "ASX_Australia_vol_20d", "LOW_Lowes_ret_5d", "spx_momentum_3d", "ENB_EnbridgeInc_ret_1d"], "is_new": true}, {"model_id": "new_h2_STRESS_LightGBM_N15_t0", "algo": "LightGBM", "regime": "STRESS", "horizon": 2, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "AMZN_ret_5d", "HD_ret_20d", "Retail_Sales_zscore_60d", "XLF_Fin_vol_20d", "EWY_Korea_zscore_60d", "CLX_Clorox_vol_20d", "DE_Deere_vol_20d", "spx_abs_ret_max_5d", "EWQ_France_ret_20d", "EFFR_ret_1d", "IWM_SmallCap_vol_20d", "HangSeng_HK_ret_5d", "NEE_NextEra_ret_20d"], "is_new": true}, {"model_id": "new_h2_STRESS_LightGBM_N15_t1", "algo": "LightGBM", "regime": "STRESS", "horizon": 2, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWG_Germany_vol_20d", "HangSeng_HK_ret_5d", "EWM_Malaysia_zscore_60d", "FedFunds_zscore_60d", "SJM_JM_Smucker_ret_5d", "LUV_SouthwestAir_ret_5d", "QQQ_vol_20d", "BTI_BritishAmerican_ret_5d", "EWG_Germany_ret_20d", "EWJ_Japan_vol_20d", "WTI_Oil_FRED_zscore_60d", "XOM_ret_1d", "Nikkei_Japan_zscore_60d"], "is_new": true}, {"model_id": "new_h2_STRESS_LightGBM_N15_t2", "algo": "LightGBM", "regime": "STRESS", "horizon": 2, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "TXN_vol_20d", "NOC_Northrop_ret_20d", "CLX_Clorox_vol_20d", "ES_Evergy_ret_1d", "SCHW_Schwab_ret_5d", "EWQ_France_zscore_60d", "HD_zscore_60d", "DAX_Germany_zscore_60d", "Core_CPI_zscore_60d", "DOW_Price_zscore_60d", "AMZN_ret_5d", "ENB_EnbridgeInc_ret_1d", "XLY_Disc_vol_20d"], "is_new": true}, {"model_id": "new_h2_STRESS_LightGBM_N15_t3", "algo": "LightGBM", "regime": "STRESS", "horizon": 2, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "AVB_AvalonBay_zscore_60d", "CMCSA_ret_1d", "HD_zscore_60d", "VRP_ma5", "HangSeng_HK_vol_20d", "US30Y_Rate_ret_20d", "ASX_Australia_ret_5d", "EMR_Emerson_ret_20d", "XLY_Disc_vol_20d", "TED_Spread_vol_20d", "XLB_Materials_zscore_60d", "AXP_Amex_vol_20d", "US1Y_Rate_ret_5d"], "is_new": true}, {"model_id": "new_h2_STRESS_LightGBM_N15_t4", "algo": "LightGBM", "regime": "STRESS", "horizon": 2, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "HangSeng_HK_ret_5d", "LOW_Lowes_ret_5d", "EFFR_vol_20d", "EWA_Australia_ret_1d", "Core_CPI_zscore_60d", "US3Y_Rate_ret_5d", "CPB_CampbellSoup_zscore_60d", "US5Y_Rate_ret_5d", "VRP_ma5", "ASX_Australia_vol_20d", "SO_SouthernCo_ret_5d", "EFFR_ret_1d", "BTI_BritishAmerican_ret_5d"], "is_new": true}, {"model_id": "new_h2_STRESS_LightGBM_N15_t5", "algo": "LightGBM", "regime": "STRESS", "horizon": 2, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "MS_MorganStanley_ret_5d", "US30Y_Rate_ret_20d", "EWG_Germany_vol_20d", "CPB_CampbellSoup_ret_20d", "PAYX_Paychex_ret_20d", "DOW_Price_zscore_60d", "EWM_Malaysia_zscore_60d", "Nikkei_Japan_zscore_60d", "XOM_ret_20d", "M_Macys_vol_20d", "SBUX_zscore_60d", "BA_ret_1d", "HangSeng_HK_ret_1d"], "is_new": true}, {"model_id": "new_h2_STRESS_LightGBM_N15_t6", "algo": "LightGBM", "regime": "STRESS", "horizon": 2, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "AMD_ret_1d", "spx_abs_ret_max_5d", "EWL_Switzerland_vol_20d", "T_ret_1d", "US3M_Rate_vol_20d", "CMCSA_ret_1d", "XLY_Disc_vol_20d", "Core_CPI_zscore_60d", "SLB_Schlumberger_ret_1d", "Retail_Sales_zscore_60d", "HD_zscore_60d", "ASX_Australia_vol_20d", "ITT_ITTInc_ret_5d"], "is_new": true}, {"model_id": "new_h2_STRESS_LightGBM_N15_t7", "algo": "LightGBM", "regime": "STRESS", "horizon": 2, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EFFR_vol_20d", "AMT_AmericanTower_ret_1d", "EWY_Korea_ret_20d", "GE_ret_1d", "FedFunds_zscore_60d", "VRP_ma5", "MO_AltriaMG_ret_1d", "Nikkei_Japan_zscore_60d", "AORD_AUS_zscore_60d", "SCHW_Schwab_ret_5d", "Brent_Oil_FRED_ret_5d", "EXC_Exelon_ret_1d", "SPY_zscore_60d"], "is_new": true}, {"model_id": "new_h2_STRESS_LightGBM_N20_t0", "algo": "LightGBM", "regime": "STRESS", "horizon": 2, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "SCHW_Schwab_ret_5d", "PCAR_PaccarInc_ret_5d", "LLY_zscore_60d", "EFFR_ret_1d", "LOW_Lowes_ret_20d", "spx_abs_ret_max_5d", "PFE_ret_1d", "DOW_Price_zscore_60d", "EWM_Malaysia_ret_1d", "Brent_Oil_FRED_ret_5d", "MSTR_Bitcoin3_ret_1d", "EWH_HongKong_ret_5d", "Core_PCE_zscore_60d", "MS_MorganStanley_ret_5d", "AMZN_ret_5d", "EWC_Canada_zscore_60d", "spx_vol_5d", "SJM_JM_Smucker_ret_1d"], "is_new": true}, {"model_id": "new_h2_STRESS_LightGBM_N20_t1", "algo": "LightGBM", "regime": "STRESS", "horizon": 2, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "INTC_ret_1d", "ORCL_zscore_60d", "AORD_AUS_zscore_60d", "EWA_Australia_ret_1d", "US3Y_Rate_ret_5d", "XLK_Tech_zscore_60d", "MSTR_Bitcoin3_ret_20d", "EWG_Germany_vol_20d", "BLK_BlackRock_zscore_60d", "NWL_Newell_ret_20d", "M_Macys_vol_20d", "PG_ret_20d", "AMT_AmericanTower_ret_1d", "AMD_ret_5d", "LLY_zscore_60d", "VVIX_ret_20d", "XLF_Fin_vol_20d", "HD_ret_1d"], "is_new": true}, {"model_id": "new_h2_STRESS_LightGBM_N20_t2", "algo": "LightGBM", "regime": "STRESS", "horizon": 2, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "DHR_ret_1d", "EWY_Korea_ret_20d", "EWL_Switzerland_vol_20d", "PG_ret_20d", "BLK_BlackRock_zscore_60d", "AORD_AUS_zscore_60d", "SCHW_Schwab_ret_5d", "CPB_CampbellSoup_zscore_60d", "INTC_ret_5d", "T10Y2Y_Spread_ret_5d", "AMD_ret_1d", "AMZN_ret_5d", "US6M_Rate_ret_20d", "BTI_BritishAmerican_ret_20d", "GILD_Gilead_ret_20d", "spx_abs_ret_max_5d", "AXP_Amex_vol_20d", "US30Y_Rate_ret_20d"], "is_new": true}, {"model_id": "new_h2_STRESS_LightGBM_N20_t3", "algo": "LightGBM", "regime": "STRESS", "horizon": 2, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWL_Switzerland_vol_20d", "HangSeng_HK_ret_1d", "AORD_AUS_zscore_60d", "Michigan_Sentiment_ret_20d", "Industrial_Production_zscore_60d", "TGT_Target_zscore_60d", "SBUX_zscore_60d", "HUM_Humana_ret_5d", "HD_ret_5d", "TM_Telephone_vol_20d", "TED_Spread_vol_20d", "US1Y_Rate_ret_20d", "EWA_Australia_zscore_60d", "XOM_ret_1d", "hmm_p_stress", "EOG_EOGResources_ret_5d", "EWS_Singapore_ret_5d", "PLD_Prologis_ret_5d"], "is_new": true}, {"model_id": "new_h2_STRESS_LightGBM_N20_t4", "algo": "LightGBM", "regime": "STRESS", "horizon": 2, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "DHR_vol_20d", "TED_Spread_vol_20d", "IWM_SmallCap_vol_20d", "TM_Telephone_vol_20d", "MRK_Merck_zscore_60d", "T_ret_1d", "Industrial_Production_zscore_60d", "GILD_Gilead_ret_20d", "LOW_Lowes_ret_20d", "INTC_ret_1d", "CTAS_Cintas_vol_20d", "EWJ_Japan_vol_20d", "XLY_Disc_vol_20d", "Core_CPI_zscore_60d", "US5Y_Rate_ret_5d", "NVDA_vol_20d", "AMZN_ret_5d", "IBEX_Spain_ret_20d"], "is_new": true}, {"model_id": "new_h2_STRESS_LightGBM_N20_t5", "algo": "LightGBM", "regime": "STRESS", "horizon": 2, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "SCHW_Schwab_ret_5d", "TM_Telephone_vol_20d", "US30Y_Rate_ret_20d", "EWG_Germany_vol_20d", "T_ret_1d", "SBUX_zscore_60d", "EWC_Canada_zscore_60d", "MS_MorganStanley_zscore_60d", "US3M_Rate_vol_20d", "heston_ev_h3", "EWA_Australia_zscore_60d", "DIS_vol_20d", "EWQ_France_zscore_60d", "TGT_Target_zscore_60d", "AMGN_Amgen_ret_1d", "ASX_Australia_ret_5d", "EXC_Exelon_zscore_60d", "EWQ_France_ret_20d"], "is_new": true}, {"model_id": "new_h2_STRESS_LightGBM_N20_t6", "algo": "LightGBM", "regime": "STRESS", "horizon": 2, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "SPY_zscore_60d", "IBEX_Spain_ret_20d", "NFCI_ret_5d", "NVDA_vol_20d", "GILD_Gilead_ret_20d", "Nikkei_Japan_zscore_60d", "Retail_Sales_zscore_60d", "GD_GeneralDynamics_zscore_60d", "CPB_CampbellSoup_ret_5d", "MS_MorganStanley_ret_5d", "EWM_Malaysia_zscore_60d", "SBUX_zscore_60d", "CLX_Clorox_vol_20d", "DE_Deere_ret_5d", "ASX_Australia_vol_20d", "M_Macys_vol_20d", "EWM_Malaysia_vol_20d", "CPB_CampbellSoup_zscore_60d"], "is_new": true}, {"model_id": "new_h2_STRESS_LightGBM_N20_t7", "algo": "LightGBM", "regime": "STRESS", "horizon": 2, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "US5Y_Rate_ret_5d", "PAYX_Paychex_ret_20d", "SBUX_ret_5d", "M_Macys_vol_20d", "ASX_Australia_vol_20d", "DAX_Germany_vol_20d", "EXC_Exelon_zscore_60d", "LOW_Lowes_ret_20d", "SO_SouthernCo_ret_5d", "HangSeng_HK_ret_5d", "3M_vol_20d", "CPB_CampbellSoup_vol_20d", "CTAS_Cintas_vol_20d", "GD_GeneralDynamics_zscore_60d", "EWY_Korea_zscore_60d", "CCI_CrownCastle_vol_20d", "MO_AltriaMG_ret_1d", "ORCL_zscore_60d"], "is_new": true}, {"model_id": "new_h2_STRESS_LightGBM_N25_t0", "algo": "LightGBM", "regime": "STRESS", "horizon": 2, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "MSTR_Bitcoin3_ret_20d", "BTI_BritishAmerican_ret_5d", "Brent_Oil_FRED_ret_20d", "NOC_Northrop_ret_20d", "TM_Telephone_ret_1d", "vix_mean_abs_ret_5d", "Core_PCE_zscore_60d", "CPB_CampbellSoup_vol_20d", "HangSeng_HK_vol_20d", "MSTR_Bitcoin3_ret_5d", "AORD_AUS_zscore_60d", "EWC_Canada_zscore_60d", "MSTR_Bitcoin3_ret_1d", "Brent_Oil_FRED_ret_5d", "CLX_Clorox_vol_20d", "SPY_zscore_60d", "XLF_Fin_vol_20d", "ENB_EnbridgeInc_ret_1d", "M_Macys_vol_20d", "AMT_AmericanTower_ret_1d", "IYR_US_REIT2_zscore_60d", "JNJ_ret_1d", "Industrial_Production_zscore_60d"], "is_new": true}, {"model_id": "new_h2_STRESS_LightGBM_N25_t1", "algo": "LightGBM", "regime": "STRESS", "horizon": 2, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EXC_Exelon_zscore_60d", "EOG_EOGResources_vol_20d", "MS_MorganStanley_zscore_60d", "SJM_JM_Smucker_ret_5d", "EWY_Korea_ret_20d", "MSTR_Bitcoin3_ret_20d", "SLB_Schlumberger_ret_1d", "EXC_Exelon_ret_1d", "EWM_Malaysia_zscore_60d", "SJM_JM_Smucker_ret_1d", "PAYX_Paychex_zscore_60d", "CLX_Clorox_vol_20d", "GD_GeneralDynamics_zscore_60d", "AMZN_ret_5d", "EWC_Canada_zscore_60d", "HD_zscore_60d", "GE_ret_1d", "INTC_ret_1d", "HD_ret_1d", "Industrial_Production_zscore_60d", "US3M_Rate_zscore_60d", "LMT_LockheedMartin_vol_20d", "NFCI_ret_5d"], "is_new": true}, {"model_id": "new_h2_STRESS_LightGBM_N25_t2", "algo": "LightGBM", "regime": "STRESS", "horizon": 2, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "3M_vol_20d", "CLX_Clorox_vol_20d", "CMCSA_ret_1d", "WTI_Oil_FRED_zscore_60d", "NFCI_ret_5d", "heston_var_ev_h5", "DHR_ret_1d", "DOW_Price_zscore_60d", "SJM_JM_Smucker_ret_1d", "hmm_p_stress", "EWL_Switzerland_zscore_60d", "T10Y2Y_Spread_ret_5d", "MSTR_Bitcoin3_ret_1d", "VOD_Vodafone_zscore_60d", "PPL_PPL_ret_1d", "GD_GeneralDynamics_zscore_60d", "DAX_Germany_zscore_60d", "CPB_CampbellSoup_ret_5d", "PAYX_Paychex_ret_20d", "EWH_HongKong_ret_5d", "MS_MorganStanley_ret_1d", "TED_Spread_zscore_60d", "EWM_Malaysia_zscore_60d"], "is_new": true}, {"model_id": "new_h2_STRESS_LightGBM_N25_t3", "algo": "LightGBM", "regime": "STRESS", "horizon": 2, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "gjr_condvar_h1", "PAYX_Paychex_zscore_60d", "AMT_AmericanTower_ret_1d", "US7Y_Rate_ret_20d", "EWA_Australia_zscore_60d", "EXC_Exelon_zscore_60d", "AORD_AUS_zscore_60d", "XLB_Materials_zscore_60d", "US3Y_Rate_ret_5d", "CI_Cigna_vol_20d", "NEE_NextEra_ret_20d", "LOW_Lowes_ret_5d", "ASX_Australia_ret_5d", "TXN_vol_20d", "Core_CPI_zscore_60d", "BTI_BritishAmerican_ret_5d", "EWA_Australia_ret_1d", "NFCI_ret_5d", "hmm_p_stress", "AMGN_Amgen_ret_1d", "EWQ_France_zscore_60d", "GILD_Gilead_ret_20d", "TED_Spread_zscore_60d"], "is_new": true}, {"model_id": "new_h2_STRESS_LightGBM_N25_t4", "algo": "LightGBM", "regime": "STRESS", "horizon": 2, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "QQQ_vol_20d", "PG_ret_20d", "ASX_Australia_ret_5d", "AMD_ret_5d", "NVDA_vol_20d", "EOG_EOGResources_vol_20d", "SBUX_vol_20d", "US3Y_Rate_ret_5d", "IYM_BasicMaterials_ret_20d", "US5Y_Rate_ret_5d", "IYR_US_REIT2_zscore_60d", "DAX_Germany_vol_20d", "HangSeng_HK_vol_20d", "EQR_Equity_ret_1d", "VVIX_ret_20d", "LMT_LockheedMartin_vol_20d", "EXC_Exelon_ret_1d", "HD_ret_20d", "MS_MorganStanley_ret_1d", "Nikkei_Japan_vol_20d", "heston_ev_h3", "heston_var_ev_h3", "HUM_Humana_ret_5d"], "is_new": true}, {"model_id": "new_h2_STRESS_LightGBM_N25_t5", "algo": "LightGBM", "regime": "STRESS", "horizon": 2, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "ORCL_vol_20d", "DE_Deere_ret_5d", "QQQ_vol_20d", "JNJ_ret_1d", "EOG_EOGResources_ret_5d", "SJM_JM_Smucker_ret_5d", "3M_vol_20d", "AMD_ret_5d", "CTAS_Cintas_vol_20d", "NOC_Northrop_ret_20d", "BTI_BritishAmerican_ret_5d", "PAYX_Paychex_vol_20d", "EWG_Germany_vol_20d", "GE_ret_1d", "HangSeng_HK_vol_20d", "PG_ret_20d", "US6M_Rate_ret_20d", "HD_ret_20d", "INTC_ret_5d", "US7Y_Rate_ret_20d", "ES_Evergy_ret_1d", "EWM_Malaysia_zscore_60d", "INTC_ret_1d"], "is_new": true}, {"model_id": "new_h2_STRESS_LightGBM_N25_t6", "algo": "LightGBM", "regime": "STRESS", "horizon": 2, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "US1Y_Rate_ret_20d", "EWM_Malaysia_vol_20d", "EMR_Emerson_ret_20d", "NWL_Newell_ret_20d", "HD_ret_1d", "SO_SouthernCo_ret_5d", "XLF_Fin_vol_20d", "SBUX_vol_20d", "CMCSA_ret_1d", "AXP_Amex_ret_20d", "BA_ret_1d", "heston_var_ev_h3", "PAYX_Paychex_ret_20d", "AMD_ret_5d", "PLD_Prologis_ret_5d", "US1Y_Rate_ret_5d", "CPB_CampbellSoup_vol_20d", "DAX_Germany_zscore_60d", "HD_ret_20d", "Industrial_Production_zscore_60d", "PAYX_Paychex_vol_20d", "Michigan_Sentiment_ret_20d", "Brent_Oil_FRED_ret_5d"], "is_new": true}, {"model_id": "new_h2_STRESS_LightGBM_N25_t7", "algo": "LightGBM", "regime": "STRESS", "horizon": 2, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "US3M_Rate_vol_20d", "HangSeng_HK_ret_5d", "US30Y_Rate_ret_20d", "HD_ret_20d", "CMCSA_ret_1d", "HD_ret_5d", "EOG_EOGResources_vol_20d", "XOM_ret_20d", "heston_var_ev_h3", "EWS_Singapore_ret_5d", "INTC_ret_1d", "LMT_LockheedMartin_ret_1d", "DOW_Price_zscore_60d", "TM_Telephone_vol_20d", "EWG_Germany_ret_20d", "EWJ_Japan_vol_20d", "SBUX_ret_5d", "EOG_EOGResources_ret_5d", "TGT_Target_zscore_60d", "PPL_PPL_ret_1d", "LOW_Lowes_ret_20d", "Michigan_Sentiment_ret_20d", "SBUX_zscore_60d"], "is_new": true}, {"model_id": "new_h2_STRESS_LightGBM_N30_t0", "algo": "LightGBM", "regime": "STRESS", "horizon": 2, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "DHR_ret_1d", "NOC_Northrop_ret_20d", "IWM_SmallCap_vol_20d", "MSTR_Bitcoin3_ret_1d", "GILD_Gilead_ret_20d", "heston_var_ev_h3", "PCAR_PaccarInc_ret_5d", "PPL_PPL_ret_1d", "CPB_CampbellSoup_ret_20d", "CI_Cigna_vol_20d", "JNJ_ret_1d", "SPY_zscore_60d", "US30Y_Rate_ret_20d", "DE_Deere_ret_5d", "BTI_BritishAmerican_ret_20d", "TED_Spread_vol_20d", "PLD_Prologis_ret_5d", "ASX_Australia_vol_20d", "T10Y2Y_Spread_ret_5d", "VVIX_ret_20d", "HD_ret_1d", "Brent_Oil_FRED_ret_20d", "3M_ret_5d", "SBUX_vol_20d", "US5Y_Rate_ret_5d", "Core_PCE_zscore_60d", "Core_CPI_zscore_60d", "vix_acceleration_1d"], "is_new": true}, {"model_id": "new_h2_STRESS_LightGBM_N30_t1", "algo": "LightGBM", "regime": "STRESS", "horizon": 2, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EXC_Exelon_ret_1d", "vix_mean_abs_ret_5d", "MSTR_Bitcoin3_ret_20d", "hmm_p_stress", "CCI_CrownCastle_vol_20d", "CPB_CampbellSoup_vol_20d", "US7Y_Rate_ret_20d", "SLB_Schlumberger_ret_1d", "EWQ_France_zscore_60d", "BLK_BlackRock_zscore_60d", "US5Y_Rate_ret_5d", "PFE_ret_1d", "LOW_Lowes_ret_5d", "HangSeng_HK_ret_5d", "EOG_EOGResources_vol_20d", "XLY_Disc_vol_20d", "SBUX_zscore_60d", "EWJ_Japan_vol_20d", "SPY_zscore_60d", "TXN_vol_20d", "Industrial_Production_zscore_60d", "ES_Evergy_ret_1d", "heston_var_ev_h3", "SJM_JM_Smucker_ret_1d", "SBUX_ret_5d", "HD_zscore_60d", "US30Y_Rate_ret_20d", "SCHW_Schwab_ret_5d"], "is_new": true}, {"model_id": "new_h2_STRESS_LightGBM_N30_t2", "algo": "LightGBM", "regime": "STRESS", "horizon": 2, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "US30Y_Rate_ret_20d", "vix_mean_abs_ret_5d", "Core_PCE_zscore_60d", "gjr_condvar_h1", "DIS_vol_20d", "JNJ_ret_1d", "EWS_Singapore_ret_5d", "EOG_EOGResources_vol_20d", "IYR_US_REIT2_zscore_60d", "CPB_CampbellSoup_ret_20d", "EFFR_vol_20d", "CLX_Clorox_vol_20d", "US1Y_Rate_ret_5d", "AORD_AUS_zscore_60d", "T_ret_1d", "EMR_Emerson_ret_20d", "ENB_EnbridgeInc_ret_1d", "Michigan_Sentiment_ret_20d", "SBUX_ret_5d", "US3Y_Rate_ret_5d", "US7Y_Rate_ret_20d", "BLK_BlackRock_zscore_60d", "NEE_NextEra_ret_20d", "XLB_Materials_zscore_60d", "NFCI_ret_5d", "AMGN_Amgen_ret_1d", "spx_abs_ret_max_5d", "TM_Telephone_vol_20d"], "is_new": true}, {"model_id": "new_h2_STRESS_LightGBM_N30_t3", "algo": "LightGBM", "regime": "STRESS", "horizon": 2, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "US3Y_Rate_ret_5d", "heston_var_ev_h7", "LOW_Lowes_ret_5d", "PFE_ret_1d", "BTI_BritishAmerican_ret_20d", "NEE_NextEra_ret_20d", "HangSeng_HK_vol_20d", "Michigan_Sentiment_ret_20d", "TED_Spread_zscore_60d", "DE_Deere_vol_20d", "CPB_CampbellSoup_zscore_60d", "3M_ret_5d", "AMD_ret_5d", "EWA_Australia_ret_1d", "PPL_PPL_ret_1d", "AMD_ret_1d", "VRP_ma5", "AMT_AmericanTower_ret_1d", "PAYX_Paychex_vol_20d", "AXP_Amex_ret_20d", "EWS_Singapore_ret_5d", "SBUX_vol_20d", "T10Y2Y_Spread_ret_5d", "BLK_BlackRock_zscore_60d", "heston_var_ev_h3", "SO_SouthernCo_ret_5d", "NOC_Northrop_ret_20d", "ORCL_vol_20d"], "is_new": true}, {"model_id": "new_h2_STRESS_LightGBM_N30_t4", "algo": "LightGBM", "regime": "STRESS", "horizon": 2, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "SJM_JM_Smucker_ret_1d", "ASX_Australia_ret_5d", "US3Y_Rate_ret_5d", "SO_SouthernCo_ret_5d", "EWQ_France_ret_20d", "GE_ret_1d", "SLB_Schlumberger_ret_1d", "hmm_p_stress", "VVIX_ret_20d", "EFFR_ret_1d", "ORCL_vol_20d", "SBUX_zscore_60d", "HangSeng_HK_ret_5d", "TM_Telephone_vol_20d", "ORCL_zscore_60d", "NEE_NextEra_ret_20d", "EQIX_Equinix_ret_5d", "XLK_Tech_zscore_60d", "LLY_zscore_60d", "BLK_BlackRock_zscore_60d", "QQQ_vol_20d", "HD_ret_5d", "GD_GeneralDynamics_zscore_60d", "JNJ_ret_1d", "MS_MorganStanley_zscore_60d", "INTC_ret_5d", "PAYX_Paychex_ret_20d", "DE_Deere_vol_20d"], "is_new": true}, {"model_id": "new_h2_STRESS_LightGBM_N30_t5", "algo": "LightGBM", "regime": "STRESS", "horizon": 2, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "GILD_Gilead_ret_20d", "CLX_Clorox_vol_20d", "WTI_Oil_FRED_zscore_60d", "SLB_Schlumberger_ret_5d", "EMR_Emerson_ret_20d", "SJM_JM_Smucker_ret_5d", "CPB_CampbellSoup_vol_20d", "PAYX_Paychex_vol_20d", "VOD_Vodafone_zscore_60d", "GE_ret_1d", "SJM_JM_Smucker_ret_1d", "Retail_Sales_zscore_60d", "US5Y_Rate_ret_5d", "LMT_LockheedMartin_ret_1d", "3M_ret_5d", "AVB_AvalonBay_zscore_60d", "EWA_Australia_zscore_60d", "HangSeng_HK_vol_20d", "IYR_US_REIT2_zscore_60d", "DE_Deere_ret_5d", "DOW_Price_zscore_60d", "BLK_BlackRock_zscore_60d", "ES_Evergy_ret_1d", "ORCL_vol_20d", "EQIX_Equinix_ret_5d", "MS_MorganStanley_zscore_60d", "EWM_Malaysia_ret_1d", "BDX_Becton_Dickinson_ret_20d"], "is_new": true}, {"model_id": "new_h2_STRESS_LightGBM_N30_t6", "algo": "LightGBM", "regime": "STRESS", "horizon": 2, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "LLY_zscore_60d", "EOG_EOGResources_ret_5d", "MRK_Merck_zscore_60d", "PAYX_Paychex_ret_20d", "EWM_Malaysia_ret_1d", "EWQ_France_zscore_60d", "AMD_ret_1d", "XLK_Tech_zscore_60d", "ASX_Australia_ret_5d", "PG_ret_20d", "EXC_Exelon_zscore_60d", "AVB_AvalonBay_zscore_60d", "MSTR_Bitcoin3_ret_20d", "EWM_Malaysia_vol_20d", "IYR_US_REIT2_zscore_60d", "CLX_Clorox_vol_20d", "Nikkei_Japan_zscore_60d", "SO_SouthernCo_ret_5d", "JNJ_ret_1d", "DIS_vol_20d", "DOW_Price_zscore_60d", "GE_ret_1d", "PLD_Prologis_ret_5d", "M_Macys_vol_20d", "Retail_Sales_zscore_60d", "MO_AltriaMG_ret_1d", "DAX_Germany_vol_20d", "DAX_Germany_zscore_60d"], "is_new": true}, {"model_id": "new_h2_STRESS_LightGBM_N30_t7", "algo": "LightGBM", "regime": "STRESS", "horizon": 2, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "BLK_BlackRock_zscore_60d", "LOW_Lowes_ret_20d", "EWH_HongKong_ret_5d", "M_Macys_vol_20d", "NEE_NextEra_ret_20d", "WTI_Oil_FRED_zscore_60d", "spx_vol_5d", "EWA_Australia_zscore_60d", "DE_Deere_vol_20d", "SCHW_Schwab_ret_5d", "ORCL_vol_20d", "EWM_Malaysia_zscore_60d", "EWL_Switzerland_vol_20d", "EXC_Exelon_zscore_60d", "DE_Deere_ret_5d", "DOW_Price_zscore_60d", "TGT_Target_zscore_60d", "LMT_LockheedMartin_vol_20d", "spx_momentum_3d", "EFFR_vol_20d", "T10Y2Y_Spread_ret_5d", "XOM_ret_20d", "PLD_Prologis_ret_5d", "DHR_ret_1d", "EWL_Switzerland_zscore_60d", "SJM_JM_Smucker_ret_1d", "EQIX_Equinix_ret_5d", "ES_Evergy_ret_1d"], "is_new": true}, {"model_id": "new_h2_STRESS_GradientBoosting_N5_t0", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 2, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWM_Malaysia_zscore_60d", "EWM_Malaysia_vol_20d", "PPL_PPL_ret_1d"], "is_new": true}, {"model_id": "new_h2_STRESS_GradientBoosting_N5_t1", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 2, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "DHR_vol_20d", "spx_abs_ret_max_5d", "EWH_HongKong_ret_5d"], "is_new": true}, {"model_id": "new_h2_STRESS_GradientBoosting_N5_t2", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 2, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "AMD_ret_1d", "GD_GeneralDynamics_zscore_60d", "EWC_Canada_zscore_60d"], "is_new": true}, {"model_id": "new_h2_STRESS_GradientBoosting_N5_t3", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 2, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "gjr_condvar_h1", "XOM_ret_20d", "TM_Telephone_ret_1d"], "is_new": true}, {"model_id": "new_h2_STRESS_GradientBoosting_N5_t4", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 2, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "M_Macys_vol_20d", "heston_ev_h3", "CI_Cigna_vol_20d"], "is_new": true}, {"model_id": "new_h2_STRESS_GradientBoosting_N5_t5", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 2, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "GD_GeneralDynamics_zscore_60d", "PG_ret_20d", "EWL_Switzerland_zscore_60d"], "is_new": true}, {"model_id": "new_h2_STRESS_GradientBoosting_N5_t6", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 2, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "Retail_Sales_zscore_60d", "NVDA_vol_20d", "US3Y_Rate_ret_5d"], "is_new": true}, {"model_id": "new_h2_STRESS_GradientBoosting_N5_t7", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 2, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "HUM_Humana_ret_5d", "MO_AltriaMG_ret_1d", "LOW_Lowes_ret_20d"], "is_new": true}, {"model_id": "new_h2_STRESS_GradientBoosting_N8_t0", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 2, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "MO_AltriaMG_ret_1d", "SBUX_zscore_60d", "vix_acceleration_1d", "TM_Telephone_vol_20d", "EWL_Switzerland_zscore_60d", "PG_ret_20d"], "is_new": true}, {"model_id": "new_h2_STRESS_GradientBoosting_N8_t1", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 2, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "AORD_AUS_zscore_60d", "ITT_ITTInc_ret_5d", "EWG_Germany_vol_20d", "DIS_vol_20d", "gjr_condvar_h1", "PAYX_Paychex_zscore_60d"], "is_new": true}, {"model_id": "new_h2_STRESS_GradientBoosting_N8_t2", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 2, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "FedFunds_zscore_60d", "DE_Deere_ret_5d", "TED_Spread_vol_20d", "US1Y_Rate_ret_5d", "EWJ_Japan_vol_20d", "heston_ev_h3"], "is_new": true}, {"model_id": "new_h2_STRESS_GradientBoosting_N8_t3", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 2, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "XLV_Health_zscore_60d", "XLF_Fin_vol_20d", "PLD_Prologis_ret_5d", "SPY_zscore_60d", "EWL_Switzerland_vol_20d", "EWJ_Japan_vol_20d"], "is_new": true}, {"model_id": "new_h2_STRESS_GradientBoosting_N8_t4", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 2, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "ENB_EnbridgeInc_ret_1d", "CPB_CampbellSoup_ret_20d", "VVIX_ret_20d", "SLB_Schlumberger_ret_5d", "MRK_Merck_zscore_60d", "US6M_Rate_ret_20d"], "is_new": true}, {"model_id": "new_h2_STRESS_GradientBoosting_N8_t5", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 2, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "CCI_CrownCastle_vol_20d", "AORD_AUS_zscore_60d", "DAX_Germany_zscore_60d", "IYM_BasicMaterials_ret_20d", "AMD_ret_1d", "MS_MorganStanley_zscore_60d"], "is_new": true}, {"model_id": "new_h2_STRESS_GradientBoosting_N8_t6", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 2, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "Michigan_Sentiment_ret_20d", "heston_var_ev_h7", "PFE_ret_1d", "EWM_Malaysia_ret_1d", "HangSeng_HK_ret_1d", "SCHW_Schwab_ret_5d"], "is_new": true}, {"model_id": "new_h2_STRESS_GradientBoosting_N8_t7", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 2, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "US3M_Rate_zscore_60d", "US5Y_Rate_ret_5d", "TGT_Target_zscore_60d", "Nikkei_Japan_vol_20d", "EWY_Korea_ret_20d", "DHR_vol_20d"], "is_new": true}, {"model_id": "new_h2_STRESS_GradientBoosting_N10_t0", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 2, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWM_Malaysia_zscore_60d", "TED_Spread_zscore_60d", "IYR_US_REIT2_zscore_60d", "ES_Evergy_ret_1d", "IBEX_Spain_ret_20d", "DIS_vol_20d", "PPL_PPL_ret_1d", "MSTR_Bitcoin3_ret_20d"], "is_new": true}, {"model_id": "new_h2_STRESS_GradientBoosting_N10_t1", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 2, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "LMT_LockheedMartin_vol_20d", "US3M_Rate_vol_20d", "M_Macys_vol_20d", "EWQ_France_zscore_60d", "PAYX_Paychex_ret_20d", "HUM_Humana_ret_5d", "EWG_Germany_vol_20d", "US3Y_Rate_ret_5d"], "is_new": true}, {"model_id": "new_h2_STRESS_GradientBoosting_N10_t2", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 2, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "XLV_Health_zscore_60d", "Industrial_Production_zscore_60d", "VVIX_ret_20d", "HD_ret_5d", "3M_vol_20d", "EWG_Germany_ret_20d", "US1Y_Rate_ret_5d", "US3M_Rate_vol_20d"], "is_new": true}, {"model_id": "new_h2_STRESS_GradientBoosting_N10_t3", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 2, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "LOW_Lowes_ret_5d", "ORCL_vol_20d", "spx_vol_5d", "spx_abs_ret_max_5d", "MO_AltriaMG_ret_1d", "EWC_Canada_zscore_60d", "Brent_Oil_FRED_ret_5d", "PPL_PPL_ret_1d"], "is_new": true}, {"model_id": "new_h2_STRESS_GradientBoosting_N10_t4", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 2, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "BA_ret_1d", "DOW_Price_zscore_60d", "CMCSA_ret_1d", "AMGN_Amgen_ret_1d", "spx_vol_5d", "DIS_vol_20d", "PLD_Prologis_ret_5d", "EWG_Germany_vol_20d"], "is_new": true}, {"model_id": "new_h2_STRESS_GradientBoosting_N10_t5", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 2, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "LMT_LockheedMartin_ret_1d", "MS_MorganStanley_ret_1d", "spx_momentum_3d", "PG_ret_20d", "EWG_Germany_vol_20d", "ES_Evergy_ret_1d", "EOG_EOGResources_ret_5d", "HD_ret_20d"], "is_new": true}, {"model_id": "new_h2_STRESS_GradientBoosting_N10_t6", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 2, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "WTI_Oil_FRED_zscore_60d", "BTI_BritishAmerican_ret_20d", "Nikkei_Japan_vol_20d", "vix_acceleration_1d", "MS_MorganStanley_ret_5d", "3M_vol_20d", "CPB_CampbellSoup_ret_20d", "US6M_Rate_ret_20d"], "is_new": true}, {"model_id": "new_h2_STRESS_GradientBoosting_N10_t7", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 2, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "T_ret_1d", "SPY_zscore_60d", "Nikkei_Japan_zscore_60d", "MS_MorganStanley_zscore_60d", "AMT_AmericanTower_ret_1d", "PAYX_Paychex_ret_20d", "EWH_HongKong_ret_5d", "EFFR_ret_1d"], "is_new": true}, {"model_id": "new_h2_STRESS_GradientBoosting_N12_t0", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 2, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "US1Y_Rate_ret_5d", "US3Y_Rate_ret_5d", "XOM_ret_20d", "EWC_Canada_zscore_60d", "vix_acceleration_1d", "PPL_PPL_ret_1d", "SO_SouthernCo_ret_5d", "LOW_Lowes_ret_20d", "XLV_Health_zscore_60d", "LOW_Lowes_ret_5d"], "is_new": true}, {"model_id": "new_h2_STRESS_GradientBoosting_N12_t1", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 2, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "Nikkei_Japan_zscore_60d", "BDX_Becton_Dickinson_ret_20d", "XLV_Health_zscore_60d", "vix_mean_abs_ret_5d", "AMT_AmericanTower_ret_1d", "WTI_Oil_FRED_zscore_60d", "EWC_Canada_zscore_60d", "SCHW_Schwab_ret_5d", "CPB_CampbellSoup_ret_20d", "NOC_Northrop_ret_20d"], "is_new": true}, {"model_id": "new_h2_STRESS_GradientBoosting_N12_t2", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 2, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "HD_ret_5d", "LMT_LockheedMartin_ret_1d", "SBUX_zscore_60d", "EWY_Korea_ret_20d", "GE_ret_1d", "NVDA_vol_20d", "ORCL_vol_20d", "US5Y_Rate_ret_5d", "HangSeng_HK_ret_1d", "EXC_Exelon_ret_1d"], "is_new": true}, {"model_id": "new_h2_STRESS_GradientBoosting_N12_t3", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 2, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "T_ret_1d", "BDX_Becton_Dickinson_ret_20d", "IWM_SmallCap_vol_20d", "DE_Deere_ret_5d", "SJM_JM_Smucker_ret_5d", "US3Y_Rate_ret_5d", "AORD_AUS_zscore_60d", "spx_momentum_3d", "IYM_BasicMaterials_ret_20d", "MS_MorganStanley_ret_1d"], "is_new": true}, {"model_id": "new_h2_STRESS_GradientBoosting_N12_t4", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 2, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "PAYX_Paychex_ret_20d", "TM_Telephone_vol_20d", "HangSeng_HK_ret_5d", "PAYX_Paychex_zscore_60d", "CMCSA_ret_1d", "VRP_ma5", "TED_Spread_zscore_60d", "SJM_JM_Smucker_ret_5d", "EWQ_France_ret_20d", "GILD_Gilead_ret_20d"], "is_new": true}, {"model_id": "new_h2_STRESS_GradientBoosting_N12_t5", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 2, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "XLF_Fin_vol_20d", "Michigan_Sentiment_ret_20d", "SLB_Schlumberger_ret_5d", "INTC_ret_5d", "XLY_Disc_vol_20d", "EWJ_Japan_vol_20d", "SJM_JM_Smucker_ret_5d", "AVB_AvalonBay_zscore_60d", "MS_MorganStanley_zscore_60d", "EFFR_vol_20d"], "is_new": true}, {"model_id": "new_h2_STRESS_GradientBoosting_N12_t6", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 2, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWQ_France_zscore_60d", "AXP_Amex_ret_20d", "SLB_Schlumberger_ret_5d", "US3Y_Rate_ret_5d", "heston_var_ev_h7", "VRP_ma5", "MS_MorganStanley_ret_1d", "HD_ret_1d", "MS_MorganStanley_ret_5d", "XLF_Fin_vol_20d"], "is_new": true}, {"model_id": "new_h2_STRESS_GradientBoosting_N12_t7", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 2, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "HangSeng_HK_ret_1d", "EWM_Malaysia_zscore_60d", "LOW_Lowes_ret_20d", "EWY_Korea_zscore_60d", "TED_Spread_zscore_60d", "Michigan_Sentiment_ret_20d", "VVIX_ret_20d", "IYM_BasicMaterials_ret_20d", "EQIX_Equinix_ret_5d", "CTAS_Cintas_vol_20d"], "is_new": true}, {"model_id": "new_h2_STRESS_GradientBoosting_N15_t0", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 2, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "ASX_Australia_ret_5d", "EWQ_France_ret_20d", "SLB_Schlumberger_ret_1d", "PAYX_Paychex_zscore_60d", "BTI_BritishAmerican_ret_20d", "3M_vol_20d", "T10Y2Y_Spread_ret_5d", "TXN_vol_20d", "AXP_Amex_ret_20d", "EWH_HongKong_ret_5d", "IYM_BasicMaterials_ret_20d", "IBEX_Spain_ret_20d", "NVDA_vol_20d"], "is_new": true}, {"model_id": "new_h2_STRESS_GradientBoosting_N15_t1", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 2, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "3M_ret_5d", "HangSeng_HK_ret_1d", "VOD_Vodafone_zscore_60d", "BLK_BlackRock_zscore_60d", "EWJ_Japan_vol_20d", "Retail_Sales_zscore_60d", "Michigan_Sentiment_ret_20d", "EXC_Exelon_zscore_60d", "ORCL_zscore_60d", "GILD_Gilead_ret_20d", "PFE_ret_1d", "Nikkei_Japan_zscore_60d", "US7Y_Rate_ret_20d"], "is_new": true}, {"model_id": "new_h2_STRESS_GradientBoosting_N15_t2", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 2, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "ES_Evergy_ret_1d", "EQIX_Equinix_ret_5d", "Core_CPI_zscore_60d", "Industrial_Production_zscore_60d", "AXP_Amex_ret_20d", "EWQ_France_ret_20d", "US5Y_Rate_ret_5d", "Michigan_Sentiment_ret_20d", "INTC_ret_1d", "LMT_LockheedMartin_ret_1d", "US1Y_Rate_ret_20d", "NEE_NextEra_ret_20d", "SJM_JM_Smucker_ret_5d"], "is_new": true}, {"model_id": "new_h2_STRESS_GradientBoosting_N15_t3", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 2, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "HangSeng_HK_ret_5d", "DAX_Germany_vol_20d", "INTC_ret_1d", "T10Y2Y_Spread_ret_5d", "XLB_Materials_zscore_60d", "EWL_Switzerland_zscore_60d", "EWL_Switzerland_vol_20d", "MRK_Merck_zscore_60d", "DE_Deere_ret_5d", "TXN_vol_20d", "NOC_Northrop_ret_20d", "spx_vol_5d", "M_Macys_vol_20d"], "is_new": true}, {"model_id": "new_h2_STRESS_GradientBoosting_N15_t4", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 2, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "INTC_ret_1d", "3M_ret_5d", "AMZN_ret_5d", "ENB_EnbridgeInc_ret_1d", "VOD_Vodafone_zscore_60d", "Industrial_Production_zscore_60d", "SBUX_ret_5d", "EWA_Australia_zscore_60d", "SPY_zscore_60d", "AMD_ret_1d", "CPB_CampbellSoup_zscore_60d", "AORD_AUS_zscore_60d", "BTI_BritishAmerican_ret_5d"], "is_new": true}, {"model_id": "new_h2_STRESS_GradientBoosting_N15_t5", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 2, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "CPB_CampbellSoup_ret_20d", "TM_Telephone_vol_20d", "CMCSA_ret_1d", "MRK_Merck_zscore_60d", "US30Y_Rate_ret_20d", "BDX_Becton_Dickinson_ret_20d", "MSTR_Bitcoin3_ret_5d", "spx_abs_ret_max_5d", "WTI_Oil_FRED_zscore_60d", "HUM_Humana_ret_5d", "XLF_Fin_vol_20d", "US3M_Rate_zscore_60d", "US6M_Rate_ret_20d"], "is_new": true}, {"model_id": "new_h2_STRESS_GradientBoosting_N15_t6", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 2, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "ASX_Australia_vol_20d", "LOW_Lowes_ret_20d", "Industrial_Production_zscore_60d", "Core_PCE_zscore_60d", "US5Y_Rate_ret_5d", "CPB_CampbellSoup_ret_20d", "vix_mean_abs_ret_5d", "VRP_ma5", "EWL_Switzerland_zscore_60d", "NFCI_ret_5d", "TGT_Target_zscore_60d", "ORCL_zscore_60d", "CCI_CrownCastle_vol_20d"], "is_new": true}, {"model_id": "new_h2_STRESS_GradientBoosting_N15_t7", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 2, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "SBUX_ret_5d", "3M_vol_20d", "TM_Telephone_ret_1d", "AMD_ret_1d", "heston_var_ev_h5", "IBEX_Spain_ret_20d", "XLK_Tech_zscore_60d", "DIS_vol_20d", "VVIX_ret_20d", "EWG_Germany_vol_20d", "EMR_Emerson_ret_20d", "US1Y_Rate_ret_5d", "LUV_SouthwestAir_ret_5d"], "is_new": true}, {"model_id": "new_h2_STRESS_GradientBoosting_N20_t0", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 2, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "CI_Cigna_vol_20d", "EFFR_ret_1d", "VRP_ma5", "EWL_Switzerland_zscore_60d", "EWH_HongKong_ret_5d", "DHR_ret_1d", "Brent_Oil_FRED_ret_5d", "EWA_Australia_zscore_60d", "HD_ret_1d", "US5Y_Rate_ret_5d", "LMT_LockheedMartin_vol_20d", "EMR_Emerson_ret_20d", "DE_Deere_vol_20d", "AVB_AvalonBay_zscore_60d", "US3M_Rate_vol_20d", "FedFunds_zscore_60d", "vix_acceleration_1d", "BA_ret_1d"], "is_new": true}, {"model_id": "new_h2_STRESS_GradientBoosting_N20_t1", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 2, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "SCHW_Schwab_ret_5d", "LOW_Lowes_ret_5d", "T10Y2Y_Spread_ret_5d", "US1Y_Rate_ret_5d", "LMT_LockheedMartin_ret_1d", "MSTR_Bitcoin3_ret_5d", "EFFR_ret_1d", "PAYX_Paychex_vol_20d", "EXC_Exelon_ret_1d", "TM_Telephone_vol_20d", "3M_vol_20d", "MS_MorganStanley_ret_1d", "M_Macys_vol_20d", "CPB_CampbellSoup_zscore_60d", "BTI_BritishAmerican_ret_5d", "EQR_Equity_ret_1d", "NOC_Northrop_ret_20d", "DE_Deere_ret_5d"], "is_new": true}, {"model_id": "new_h2_STRESS_GradientBoosting_N20_t2", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 2, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "MS_MorganStanley_ret_1d", "VRP_ma5", "MO_AltriaMG_ret_1d", "Brent_Oil_FRED_ret_20d", "Nikkei_Japan_vol_20d", "INTC_ret_5d", "SBUX_zscore_60d", "CMCSA_ret_1d", "EOG_EOGResources_ret_5d", "CTAS_Cintas_vol_20d", "Core_PCE_zscore_60d", "US1Y_Rate_ret_20d", "Nikkei_Japan_zscore_60d", "AXP_Amex_vol_20d", "heston_var_ev_h5", "BTI_BritishAmerican_ret_20d", "TM_Telephone_vol_20d", "CCI_CrownCastle_vol_20d"], "is_new": true}, {"model_id": "new_h2_STRESS_GradientBoosting_N20_t3", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 2, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWC_Canada_zscore_60d", "BLK_BlackRock_zscore_60d", "EQR_Equity_ret_1d", "IWM_SmallCap_vol_20d", "LOW_Lowes_ret_20d", "US7Y_Rate_ret_20d", "SLB_Schlumberger_ret_5d", "SCHW_Schwab_ret_5d", "US6M_Rate_ret_20d", "DE_Deere_vol_20d", "Nikkei_Japan_vol_20d", "DOW_Price_zscore_60d", "EWL_Switzerland_zscore_60d", "Core_CPI_zscore_60d", "HD_ret_20d", "spx_vol_5d", "JNJ_ret_1d", "WTI_Oil_FRED_zscore_60d"], "is_new": true}, {"model_id": "new_h2_STRESS_GradientBoosting_N20_t4", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 2, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "heston_ev_h3", "CPB_CampbellSoup_zscore_60d", "US3Y_Rate_ret_5d", "Michigan_Sentiment_ret_20d", "T_ret_1d", "AXP_Amex_vol_20d", "HD_ret_5d", "LOW_Lowes_ret_20d", "AORD_AUS_zscore_60d", "LLY_zscore_60d", "EMR_Emerson_ret_20d", "CLX_Clorox_vol_20d", "BTI_BritishAmerican_ret_20d", "DHR_vol_20d", "EWL_Switzerland_vol_20d", "NWL_Newell_ret_20d", "3M_vol_20d", "SPY_zscore_60d"], "is_new": true}, {"model_id": "new_h2_STRESS_GradientBoosting_N20_t5", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 2, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "US1Y_Rate_ret_5d", "AMD_ret_5d", "EWY_Korea_zscore_60d", "EXC_Exelon_ret_1d", "NEE_NextEra_ret_20d", "AVB_AvalonBay_zscore_60d", "TM_Telephone_vol_20d", "Brent_Oil_FRED_ret_5d", "XLK_Tech_zscore_60d", "SJM_JM_Smucker_ret_1d", "CPB_CampbellSoup_vol_20d", "ASX_Australia_vol_20d", "US1Y_Rate_ret_20d", "EWQ_France_ret_20d", "GE_ret_1d", "NWL_Newell_ret_20d", "US3M_Rate_vol_20d", "IWM_SmallCap_vol_20d"], "is_new": true}, {"model_id": "new_h2_STRESS_GradientBoosting_N20_t6", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 2, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "Michigan_Sentiment_ret_20d", "AMD_ret_1d", "VRP_ma5", "IYR_US_REIT2_zscore_60d", "SJM_JM_Smucker_ret_5d", "ASX_Australia_ret_5d", "NFCI_ret_5d", "DE_Deere_ret_5d", "PCAR_PaccarInc_ret_5d", "FedFunds_zscore_60d", "JNJ_ret_1d", "MS_MorganStanley_zscore_60d", "Nikkei_Japan_zscore_60d", "VVIX_ret_20d", "US1Y_Rate_ret_5d", "EWL_Switzerland_zscore_60d", "XLY_Disc_vol_20d", "heston_var_ev_h3"], "is_new": true}, {"model_id": "new_h2_STRESS_GradientBoosting_N20_t7", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 2, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EXC_Exelon_zscore_60d", "DHR_vol_20d", "XLF_Fin_vol_20d", "GILD_Gilead_ret_20d", "CCI_CrownCastle_vol_20d", "HangSeng_HK_vol_20d", "vix_acceleration_1d", "XLY_Disc_vol_20d", "spx_momentum_3d", "AMGN_Amgen_ret_1d", "HD_zscore_60d", "LOW_Lowes_ret_20d", "PCAR_PaccarInc_ret_5d", "hmm_p_stress", "PAYX_Paychex_ret_20d", "Nikkei_Japan_zscore_60d", "EQIX_Equinix_ret_5d", "INTC_ret_1d"], "is_new": true}, {"model_id": "new_h2_STRESS_GradientBoosting_N25_t0", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 2, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "MS_MorganStanley_ret_1d", "XLB_Materials_zscore_60d", "CI_Cigna_vol_20d", "WTI_Oil_FRED_zscore_60d", "LLY_zscore_60d", "EWA_Australia_zscore_60d", "AXP_Amex_ret_20d", "CPB_CampbellSoup_ret_20d", "MS_MorganStanley_ret_5d", "HD_ret_5d", "SCHW_Schwab_ret_5d", "DHR_ret_1d", "TED_Spread_vol_20d", "M_Macys_vol_20d", "DAX_Germany_vol_20d", "PPL_PPL_ret_1d", "HUM_Humana_ret_5d", "MSTR_Bitcoin3_ret_1d", "SPY_zscore_60d", "INTC_ret_5d", "US7Y_Rate_ret_20d", "AMD_ret_5d", "DE_Deere_vol_20d"], "is_new": true}, {"model_id": "new_h2_STRESS_GradientBoosting_N25_t1", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 2, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "MS_MorganStanley_ret_1d", "CI_Cigna_vol_20d", "Core_PCE_zscore_60d", "vix_mean_abs_ret_5d", "spx_momentum_3d", "DHR_ret_1d", "TM_Telephone_vol_20d", "M_Macys_vol_20d", "ORCL_vol_20d", "AMD_ret_1d", "EWA_Australia_ret_1d", "HangSeng_HK_vol_20d", "VVIX_ret_20d", "T10Y2Y_Spread_ret_5d", "DAX_Germany_zscore_60d", "GE_ret_1d", "heston_var_ev_h5", "T_ret_1d", "XOM_ret_1d", "NFCI_ret_5d", "SBUX_ret_5d", "AXP_Amex_ret_20d", "DAX_Germany_vol_20d"], "is_new": true}, {"model_id": "new_h2_STRESS_GradientBoosting_N25_t2", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 2, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "SJM_JM_Smucker_ret_1d", "SBUX_zscore_60d", "HUM_Humana_ret_5d", "AXP_Amex_ret_20d", "EFFR_vol_20d", "BTI_BritishAmerican_ret_20d", "LUV_SouthwestAir_ret_5d", "vix_mean_abs_ret_5d", "DE_Deere_ret_5d", "EWG_Germany_vol_20d", "CLX_Clorox_vol_20d", "DIS_vol_20d", "ITT_ITTInc_ret_5d", "XOM_ret_20d", "GILD_Gilead_ret_20d", "EQIX_Equinix_ret_5d", "EWG_Germany_ret_20d", "JNJ_ret_1d", "DOW_Price_zscore_60d", "XLB_Materials_zscore_60d", "MS_MorganStanley_ret_1d", "ASX_Australia_ret_5d", "3M_ret_5d"], "is_new": true}, {"model_id": "new_h2_STRESS_GradientBoosting_N25_t3", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 2, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "BTI_BritishAmerican_ret_20d", "PAYX_Paychex_zscore_60d", "EQR_Equity_ret_1d", "CPB_CampbellSoup_zscore_60d", "LOW_Lowes_ret_5d", "BDX_Becton_Dickinson_ret_20d", "Industrial_Production_zscore_60d", "DOW_Price_zscore_60d", "AMD_ret_1d", "XOM_ret_20d", "spx_abs_ret_max_5d", "EWM_Malaysia_vol_20d", "IYM_BasicMaterials_ret_20d", "EWC_Canada_zscore_60d", "CLX_Clorox_vol_20d", "EWA_Australia_ret_1d", "Core_PCE_zscore_60d", "SO_SouthernCo_ret_5d", "GE_ret_1d", "MRK_Merck_zscore_60d", "NFCI_ret_5d", "AMGN_Amgen_ret_1d", "XOM_ret_1d"], "is_new": true}, {"model_id": "new_h2_STRESS_GradientBoosting_N25_t4", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 2, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWS_Singapore_ret_5d", "DOW_Price_zscore_60d", "DAX_Germany_zscore_60d", "DAX_Germany_vol_20d", "3M_vol_20d", "XLB_Materials_zscore_60d", "CI_Cigna_vol_20d", "DIS_vol_20d", "US6M_Rate_ret_20d", "GE_ret_1d", "EXC_Exelon_zscore_60d", "EWM_Malaysia_ret_1d", "CPB_CampbellSoup_zscore_60d", "GILD_Gilead_ret_20d", "Brent_Oil_FRED_ret_5d", "HD_ret_1d", "AORD_AUS_zscore_60d", "TM_Telephone_ret_1d", "SBUX_vol_20d", "SO_SouthernCo_ret_5d", "EWL_Switzerland_zscore_60d", "FedFunds_zscore_60d", "EWC_Canada_zscore_60d"], "is_new": true}, {"model_id": "new_h2_STRESS_GradientBoosting_N25_t5", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 2, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "HangSeng_HK_ret_5d", "AXP_Amex_ret_20d", "DOW_Price_zscore_60d", "CPB_CampbellSoup_ret_20d", "Core_CPI_zscore_60d", "Nikkei_Japan_vol_20d", "TED_Spread_zscore_60d", "DIS_vol_20d", "EWM_Malaysia_vol_20d", "CMCSA_ret_1d", "EWL_Switzerland_vol_20d", "SBUX_zscore_60d", "XOM_ret_1d", "US3M_Rate_vol_20d", "SPY_zscore_60d", "CCI_CrownCastle_vol_20d", "SJM_JM_Smucker_ret_5d", "CPB_CampbellSoup_zscore_60d", "AMD_ret_5d", "AVB_AvalonBay_zscore_60d", "US3M_Rate_zscore_60d", "MSTR_Bitcoin3_ret_5d", "spx_vol_5d"], "is_new": true}, {"model_id": "new_h2_STRESS_GradientBoosting_N25_t6", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 2, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "PG_ret_20d", "SPY_zscore_60d", "XLV_Health_zscore_60d", "ORCL_vol_20d", "EWL_Switzerland_vol_20d", "IYR_US_REIT2_zscore_60d", "LOW_Lowes_ret_5d", "SLB_Schlumberger_ret_1d", "QQQ_vol_20d", "PAYX_Paychex_zscore_60d", "HD_ret_20d", "LLY_zscore_60d", "DAX_Germany_zscore_60d", "NOC_Northrop_ret_20d", "EWJ_Japan_vol_20d", "HangSeng_HK_ret_1d", "SJM_JM_Smucker_ret_5d", "T10Y2Y_Spread_ret_5d", "TGT_Target_zscore_60d", "DHR_vol_20d", "EWC_Canada_zscore_60d", "SCHW_Schwab_ret_5d", "DOW_Price_zscore_60d"], "is_new": true}, {"model_id": "new_h2_STRESS_GradientBoosting_N25_t7", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 2, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "DAX_Germany_zscore_60d", "heston_var_ev_h3", "ITT_ITTInc_ret_5d", "EWG_Germany_ret_20d", "IWM_SmallCap_vol_20d", "Core_CPI_zscore_60d", "BTI_BritishAmerican_ret_5d", "PLD_Prologis_ret_5d", "TED_Spread_vol_20d", "US1Y_Rate_ret_5d", "ES_Evergy_ret_1d", "US1Y_Rate_ret_20d", "AMZN_ret_5d", "IYM_BasicMaterials_ret_20d", "AXP_Amex_ret_20d", "EWY_Korea_ret_20d", "EWY_Korea_zscore_60d", "Industrial_Production_zscore_60d", "LUV_SouthwestAir_ret_5d", "EWM_Malaysia_vol_20d", "XOM_ret_20d", "vix_acceleration_1d", "gjr_condvar_h1"], "is_new": true}, {"model_id": "new_h2_STRESS_GradientBoosting_N30_t0", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 2, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "FedFunds_zscore_60d", "ORCL_zscore_60d", "MS_MorganStanley_zscore_60d", "QQQ_vol_20d", "ORCL_vol_20d", "MSTR_Bitcoin3_ret_20d", "MSTR_Bitcoin3_ret_1d", "AXP_Amex_ret_20d", "TGT_Target_zscore_60d", "DAX_Germany_zscore_60d", "TM_Telephone_vol_20d", "TXN_vol_20d", "SCHW_Schwab_ret_5d", "HangSeng_HK_ret_1d", "ITT_ITTInc_ret_5d", "SJM_JM_Smucker_ret_5d", "PAYX_Paychex_vol_20d", "CCI_CrownCastle_vol_20d", "DE_Deere_ret_5d", "BDX_Becton_Dickinson_ret_20d", "PPL_PPL_ret_1d", "INTC_ret_1d", "EWH_HongKong_ret_5d", "Brent_Oil_FRED_ret_20d", "EWL_Switzerland_vol_20d", "MS_MorganStanley_ret_1d", "US1Y_Rate_ret_5d", "LOW_Lowes_ret_5d"], "is_new": true}, {"model_id": "new_h2_STRESS_GradientBoosting_N30_t1", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 2, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "DE_Deere_vol_20d", "MS_MorganStanley_zscore_60d", "DAX_Germany_zscore_60d", "CPB_CampbellSoup_ret_20d", "spx_abs_ret_max_5d", "ASX_Australia_ret_5d", "SBUX_ret_5d", "HUM_Humana_ret_5d", "CI_Cigna_vol_20d", "EXC_Exelon_ret_1d", "FedFunds_zscore_60d", "EWL_Switzerland_zscore_60d", "3M_ret_5d", "DAX_Germany_vol_20d", "AMD_ret_5d", "MS_MorganStanley_ret_1d", "CCI_CrownCastle_vol_20d", "HD_ret_1d", "HD_ret_5d", "LOW_Lowes_ret_20d", "DHR_vol_20d", "MSTR_Bitcoin3_ret_20d", "HangSeng_HK_vol_20d", "LMT_LockheedMartin_vol_20d", "PG_ret_20d", "IBEX_Spain_ret_20d", "GE_ret_1d", "EWG_Germany_vol_20d"], "is_new": true}, {"model_id": "new_h2_STRESS_GradientBoosting_N30_t2", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 2, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "DAX_Germany_vol_20d", "LOW_Lowes_ret_5d", "EWM_Malaysia_vol_20d", "spx_vol_5d", "EWS_Singapore_ret_5d", "GILD_Gilead_ret_20d", "US6M_Rate_ret_20d", "PAYX_Paychex_zscore_60d", "ENB_EnbridgeInc_ret_1d", "XOM_ret_1d", "Nikkei_Japan_zscore_60d", "XLV_Health_zscore_60d", "PLD_Prologis_ret_5d", "AORD_AUS_zscore_60d", "CMCSA_ret_1d", "AMZN_ret_5d", "AXP_Amex_ret_20d", "EWG_Germany_vol_20d", "HangSeng_HK_ret_5d", "EWM_Malaysia_ret_1d", "HUM_Humana_ret_5d", "WTI_Oil_FRED_zscore_60d", "SJM_JM_Smucker_ret_1d", "LUV_SouthwestAir_ret_5d", "TM_Telephone_vol_20d", "LOW_Lowes_ret_20d", "US1Y_Rate_ret_20d", "NFCI_ret_5d"], "is_new": true}, {"model_id": "new_h2_STRESS_GradientBoosting_N30_t3", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 2, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "SO_SouthernCo_ret_5d", "Michigan_Sentiment_ret_20d", "BA_ret_1d", "DE_Deere_ret_5d", "Core_CPI_zscore_60d", "LUV_SouthwestAir_ret_5d", "US3Y_Rate_ret_5d", "VRP_ma5", "EFFR_ret_1d", "SJM_JM_Smucker_ret_1d", "XLF_Fin_vol_20d", "EXC_Exelon_ret_1d", "EWL_Switzerland_vol_20d", "PCAR_PaccarInc_ret_5d", "ORCL_vol_20d", "EWL_Switzerland_zscore_60d", "vix_mean_abs_ret_5d", "DAX_Germany_zscore_60d", "SBUX_zscore_60d", "HD_ret_5d", "US7Y_Rate_ret_20d", "AXP_Amex_ret_20d", "EWH_HongKong_ret_5d", "EWJ_Japan_vol_20d", "Core_PCE_zscore_60d", "EMR_Emerson_ret_20d", "ENB_EnbridgeInc_ret_1d", "ES_Evergy_ret_1d"], "is_new": true}, {"model_id": "new_h2_STRESS_GradientBoosting_N30_t4", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 2, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "ORCL_zscore_60d", "AMD_ret_5d", "HD_zscore_60d", "PAYX_Paychex_zscore_60d", "EWG_Germany_vol_20d", "Core_PCE_zscore_60d", "IBEX_Spain_ret_20d", "TED_Spread_vol_20d", "EWL_Switzerland_vol_20d", "MS_MorganStanley_zscore_60d", "US1Y_Rate_ret_5d", "PG_ret_20d", "heston_var_ev_h7", "PFE_ret_1d", "PPL_PPL_ret_1d", "SJM_JM_Smucker_ret_1d", "EXC_Exelon_ret_1d", "3M_vol_20d", "XLY_Disc_vol_20d", "AMGN_Amgen_ret_1d", "EWL_Switzerland_zscore_60d", "Michigan_Sentiment_ret_20d", "MSTR_Bitcoin3_ret_5d", "NFCI_ret_5d", "LMT_LockheedMartin_vol_20d", "M_Macys_vol_20d", "vix_acceleration_1d", "3M_ret_5d"], "is_new": true}, {"model_id": "new_h2_STRESS_GradientBoosting_N30_t5", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 2, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EMR_Emerson_ret_20d", "T_ret_1d", "EFFR_ret_1d", "Core_PCE_zscore_60d", "EQR_Equity_ret_1d", "heston_var_ev_h7", "TM_Telephone_ret_1d", "DAX_Germany_zscore_60d", "spx_momentum_3d", "US6M_Rate_ret_20d", "EWJ_Japan_vol_20d", "SJM_JM_Smucker_ret_5d", "NOC_Northrop_ret_20d", "hmm_p_stress", "ENB_EnbridgeInc_ret_1d", "SBUX_ret_5d", "US1Y_Rate_ret_5d", "EXC_Exelon_zscore_60d", "ORCL_zscore_60d", "DE_Deere_vol_20d", "Nikkei_Japan_vol_20d", "XLB_Materials_zscore_60d", "EWL_Switzerland_vol_20d", "EWG_Germany_vol_20d", "EWC_Canada_zscore_60d", "AMD_ret_1d", "HangSeng_HK_ret_1d", "LOW_Lowes_ret_5d"], "is_new": true}, {"model_id": "new_h2_STRESS_GradientBoosting_N30_t6", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 2, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "M_Macys_vol_20d", "vix_mean_abs_ret_5d", "spx_momentum_3d", "MS_MorganStanley_zscore_60d", "ASX_Australia_ret_5d", "GE_ret_1d", "TXN_vol_20d", "LMT_LockheedMartin_vol_20d", "EWM_Malaysia_zscore_60d", "CPB_CampbellSoup_ret_20d", "EQR_Equity_ret_1d", "EWG_Germany_ret_20d", "EWY_Korea_zscore_60d", "FedFunds_zscore_60d", "EWG_Germany_vol_20d", "JNJ_ret_1d", "T10Y2Y_Spread_ret_5d", "GILD_Gilead_ret_20d", "US6M_Rate_ret_20d", "US30Y_Rate_ret_20d", "CMCSA_ret_1d", "CTAS_Cintas_vol_20d", "XLF_Fin_vol_20d", "IBEX_Spain_ret_20d", "Nikkei_Japan_zscore_60d", "LMT_LockheedMartin_ret_1d", "CPB_CampbellSoup_ret_5d", "EFFR_ret_1d"], "is_new": true}, {"model_id": "new_h2_STRESS_GradientBoosting_N30_t7", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 2, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "MS_MorganStanley_ret_1d", "EWQ_France_ret_20d", "PLD_Prologis_ret_5d", "US1Y_Rate_ret_5d", "CLX_Clorox_vol_20d", "QQQ_vol_20d", "AMGN_Amgen_ret_1d", "HangSeng_HK_vol_20d", "US30Y_Rate_ret_20d", "EWM_Malaysia_ret_1d", "MSTR_Bitcoin3_ret_1d", "EWY_Korea_zscore_60d", "TED_Spread_zscore_60d", "CPB_CampbellSoup_ret_20d", "CI_Cigna_vol_20d", "NFCI_ret_5d", "Industrial_Production_zscore_60d", "US6M_Rate_ret_20d", "LMT_LockheedMartin_ret_1d", "LOW_Lowes_ret_5d", "US1Y_Rate_ret_20d", "MRK_Merck_zscore_60d", "US3Y_Rate_ret_5d", "EXC_Exelon_zscore_60d", "ENB_EnbridgeInc_ret_1d", "Brent_Oil_FRED_ret_20d", "EWY_Korea_ret_20d", "SJM_JM_Smucker_ret_1d"], "is_new": true}, {"model_id": "new_h2_STRESS_RandomForest_N5_t0", "algo": "RandomForest", "regime": "STRESS", "horizon": 2, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "NEE_NextEra_ret_20d", "EWA_Australia_zscore_60d", "MS_MorganStanley_ret_5d"], "is_new": true}, {"model_id": "new_h2_STRESS_RandomForest_N5_t1", "algo": "RandomForest", "regime": "STRESS", "horizon": 2, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "M_Macys_vol_20d", "PG_ret_20d", "Nikkei_Japan_vol_20d"], "is_new": true}, {"model_id": "new_h2_STRESS_RandomForest_N5_t2", "algo": "RandomForest", "regime": "STRESS", "horizon": 2, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "LUV_SouthwestAir_ret_5d", "US3M_Rate_zscore_60d", "EWH_HongKong_ret_5d"], "is_new": true}, {"model_id": "new_h2_STRESS_RandomForest_N5_t3", "algo": "RandomForest", "regime": "STRESS", "horizon": 2, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "IBEX_Spain_ret_20d", "CLX_Clorox_vol_20d", "SBUX_vol_20d"], "is_new": true}, {"model_id": "new_h2_STRESS_RandomForest_N5_t4", "algo": "RandomForest", "regime": "STRESS", "horizon": 2, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "US1Y_Rate_ret_5d", "AMZN_ret_5d", "T_ret_1d"], "is_new": true}, {"model_id": "new_h2_STRESS_RandomForest_N5_t5", "algo": "RandomForest", "regime": "STRESS", "horizon": 2, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "SBUX_zscore_60d", "DE_Deere_ret_5d", "EWA_Australia_zscore_60d"], "is_new": true}, {"model_id": "new_h2_STRESS_RandomForest_N5_t6", "algo": "RandomForest", "regime": "STRESS", "horizon": 2, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "Core_CPI_zscore_60d", "LUV_SouthwestAir_ret_5d", "AORD_AUS_zscore_60d"], "is_new": true}, {"model_id": "new_h2_STRESS_RandomForest_N5_t7", "algo": "RandomForest", "regime": "STRESS", "horizon": 2, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWM_Malaysia_ret_1d", "XLK_Tech_zscore_60d", "CPB_CampbellSoup_zscore_60d"], "is_new": true}, {"model_id": "new_h2_STRESS_RandomForest_N8_t0", "algo": "RandomForest", "regime": "STRESS", "horizon": 2, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "HD_ret_20d", "EQR_Equity_ret_1d", "EWC_Canada_zscore_60d", "NFCI_ret_5d", "EWY_Korea_zscore_60d", "SJM_JM_Smucker_ret_1d"], "is_new": true}, {"model_id": "new_h2_STRESS_RandomForest_N8_t1", "algo": "RandomForest", "regime": "STRESS", "horizon": 2, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "US7Y_Rate_ret_20d", "XLB_Materials_zscore_60d", "AMD_ret_5d", "CLX_Clorox_vol_20d", "3M_ret_5d", "TED_Spread_vol_20d"], "is_new": true}, {"model_id": "new_h2_STRESS_RandomForest_N8_t2", "algo": "RandomForest", "regime": "STRESS", "horizon": 2, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "PFE_ret_1d", "XLF_Fin_vol_20d", "AMGN_Amgen_ret_1d", "IBEX_Spain_ret_20d", "XOM_ret_20d", "spx_momentum_3d"], "is_new": true}, {"model_id": "new_h2_STRESS_RandomForest_N8_t3", "algo": "RandomForest", "regime": "STRESS", "horizon": 2, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWM_Malaysia_ret_1d", "TED_Spread_zscore_60d", "spx_vol_5d", "AMZN_ret_5d", "MSTR_Bitcoin3_ret_20d", "PCAR_PaccarInc_ret_5d"], "is_new": true}, {"model_id": "new_h2_STRESS_RandomForest_N8_t4", "algo": "RandomForest", "regime": "STRESS", "horizon": 2, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "MS_MorganStanley_ret_1d", "PAYX_Paychex_zscore_60d", "EQR_Equity_ret_1d", "ASX_Australia_vol_20d", "US7Y_Rate_ret_20d", "MS_MorganStanley_ret_5d"], "is_new": true}, {"model_id": "new_h2_STRESS_RandomForest_N8_t5", "algo": "RandomForest", "regime": "STRESS", "horizon": 2, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "PFE_ret_1d", "EQR_Equity_ret_1d", "Nikkei_Japan_zscore_60d", "HangSeng_HK_vol_20d", "EWM_Malaysia_ret_1d", "EOG_EOGResources_vol_20d"], "is_new": true}, {"model_id": "new_h2_STRESS_RandomForest_N8_t6", "algo": "RandomForest", "regime": "STRESS", "horizon": 2, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EFFR_vol_20d", "GD_GeneralDynamics_zscore_60d", "LOW_Lowes_ret_5d", "XLY_Disc_vol_20d", "PPL_PPL_ret_1d", "CPB_CampbellSoup_vol_20d"], "is_new": true}, {"model_id": "new_h2_STRESS_RandomForest_N8_t7", "algo": "RandomForest", "regime": "STRESS", "horizon": 2, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "XLY_Disc_vol_20d", "EWC_Canada_zscore_60d", "EFFR_vol_20d", "SBUX_zscore_60d", "PLD_Prologis_ret_5d", "3M_vol_20d"], "is_new": true}, {"model_id": "new_h2_STRESS_RandomForest_N10_t0", "algo": "RandomForest", "regime": "STRESS", "horizon": 2, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "XLY_Disc_vol_20d", "CPB_CampbellSoup_ret_20d", "PLD_Prologis_ret_5d", "SJM_JM_Smucker_ret_1d", "EWG_Germany_vol_20d", "XOM_ret_20d", "EQR_Equity_ret_1d", "NFCI_ret_5d"], "is_new": true}, {"model_id": "new_h2_STRESS_RandomForest_N10_t1", "algo": "RandomForest", "regime": "STRESS", "horizon": 2, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "SLB_Schlumberger_ret_5d", "US1Y_Rate_ret_20d", "TM_Telephone_ret_1d", "heston_var_ev_h5", "T_ret_1d", "NVDA_vol_20d", "BTI_BritishAmerican_ret_20d", "ASX_Australia_ret_5d"], "is_new": true}, {"model_id": "new_h2_STRESS_RandomForest_N10_t2", "algo": "RandomForest", "regime": "STRESS", "horizon": 2, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "LLY_zscore_60d", "SJM_JM_Smucker_ret_5d", "MSTR_Bitcoin3_ret_1d", "Retail_Sales_zscore_60d", "VRP_ma5", "HUM_Humana_ret_5d", "EWJ_Japan_vol_20d", "EWA_Australia_zscore_60d"], "is_new": true}, {"model_id": "new_h2_STRESS_RandomForest_N10_t3", "algo": "RandomForest", "regime": "STRESS", "horizon": 2, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "BA_ret_1d", "EOG_EOGResources_ret_5d", "MSTR_Bitcoin3_ret_20d", "EWG_Germany_ret_20d", "DOW_Price_zscore_60d", "SBUX_zscore_60d", "MS_MorganStanley_zscore_60d", "SCHW_Schwab_ret_5d"], "is_new": true}, {"model_id": "new_h2_STRESS_RandomForest_N10_t4", "algo": "RandomForest", "regime": "STRESS", "horizon": 2, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "FedFunds_zscore_60d", "IYR_US_REIT2_zscore_60d", "SPY_zscore_60d", "PAYX_Paychex_ret_20d", "spx_abs_ret_max_5d", "MS_MorganStanley_ret_1d", "MSTR_Bitcoin3_ret_5d", "XLY_Disc_vol_20d"], "is_new": true}, {"model_id": "new_h2_STRESS_RandomForest_N10_t5", "algo": "RandomForest", "regime": "STRESS", "horizon": 2, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "LMT_LockheedMartin_ret_1d", "LMT_LockheedMartin_vol_20d", "XOM_ret_20d", "EWQ_France_zscore_60d", "GILD_Gilead_ret_20d", "PAYX_Paychex_ret_20d", "EWY_Korea_zscore_60d", "INTC_ret_5d"], "is_new": true}, {"model_id": "new_h2_STRESS_RandomForest_N10_t6", "algo": "RandomForest", "regime": "STRESS", "horizon": 2, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "AMZN_ret_5d", "LUV_SouthwestAir_ret_5d", "EWQ_France_zscore_60d", "PAYX_Paychex_ret_20d", "SCHW_Schwab_ret_5d", "EOG_EOGResources_ret_5d", "WTI_Oil_FRED_zscore_60d", "heston_var_ev_h7"], "is_new": true}, {"model_id": "new_h2_STRESS_RandomForest_N10_t7", "algo": "RandomForest", "regime": "STRESS", "horizon": 2, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWM_Malaysia_vol_20d", "heston_var_ev_h5", "XOM_ret_1d", "EXC_Exelon_zscore_60d", "IYR_US_REIT2_zscore_60d", "AMZN_ret_5d", "EWQ_France_ret_20d", "ORCL_zscore_60d"], "is_new": true}, {"model_id": "new_h2_STRESS_RandomForest_N12_t0", "algo": "RandomForest", "regime": "STRESS", "horizon": 2, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "TED_Spread_vol_20d", "3M_vol_20d", "MRK_Merck_zscore_60d", "EWM_Malaysia_vol_20d", "TGT_Target_zscore_60d", "EWL_Switzerland_vol_20d", "EWC_Canada_zscore_60d", "Michigan_Sentiment_ret_20d", "US6M_Rate_ret_20d", "EWA_Australia_ret_1d"], "is_new": true}, {"model_id": "new_h2_STRESS_RandomForest_N12_t1", "algo": "RandomForest", "regime": "STRESS", "horizon": 2, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "WTI_Oil_FRED_zscore_60d", "MSTR_Bitcoin3_ret_20d", "HangSeng_HK_ret_5d", "TED_Spread_vol_20d", "AVB_AvalonBay_zscore_60d", "GILD_Gilead_ret_20d", "AORD_AUS_zscore_60d", "QQQ_vol_20d", "EWL_Switzerland_zscore_60d", "PFE_ret_1d"], "is_new": true}, {"model_id": "new_h2_STRESS_RandomForest_N12_t2", "algo": "RandomForest", "regime": "STRESS", "horizon": 2, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "LMT_LockheedMartin_vol_20d", "3M_ret_5d", "MSTR_Bitcoin3_ret_5d", "EWL_Switzerland_vol_20d", "EQIX_Equinix_ret_5d", "EXC_Exelon_ret_1d", "AORD_AUS_zscore_60d", "INTC_ret_1d", "CPB_CampbellSoup_vol_20d", "AXP_Amex_ret_20d"], "is_new": true}, {"model_id": "new_h2_STRESS_RandomForest_N12_t3", "algo": "RandomForest", "regime": "STRESS", "horizon": 2, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "SPY_zscore_60d", "EWM_Malaysia_vol_20d", "TGT_Target_zscore_60d", "PAYX_Paychex_zscore_60d", "gjr_condvar_h1", "US6M_Rate_ret_20d", "Nikkei_Japan_zscore_60d", "EWG_Germany_vol_20d", "EWG_Germany_ret_20d", "HD_zscore_60d"], "is_new": true}, {"model_id": "new_h2_STRESS_RandomForest_N12_t4", "algo": "RandomForest", "regime": "STRESS", "horizon": 2, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "GD_GeneralDynamics_zscore_60d", "MS_MorganStanley_ret_5d", "Nikkei_Japan_zscore_60d", "ASX_Australia_vol_20d", "EWY_Korea_ret_20d", "EWL_Switzerland_vol_20d", "DE_Deere_vol_20d", "SBUX_vol_20d", "EWC_Canada_zscore_60d", "US3Y_Rate_ret_5d"], "is_new": true}, {"model_id": "new_h2_STRESS_RandomForest_N12_t5", "algo": "RandomForest", "regime": "STRESS", "horizon": 2, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "IYR_US_REIT2_zscore_60d", "VVIX_ret_20d", "DIS_vol_20d", "BA_ret_1d", "US5Y_Rate_ret_5d", "HD_ret_20d", "BTI_BritishAmerican_ret_5d", "CLX_Clorox_vol_20d", "AMT_AmericanTower_ret_1d", "EQIX_Equinix_ret_5d"], "is_new": true}, {"model_id": "new_h2_STRESS_RandomForest_N12_t6", "algo": "RandomForest", "regime": "STRESS", "horizon": 2, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "SBUX_ret_5d", "TGT_Target_zscore_60d", "CTAS_Cintas_vol_20d", "VVIX_ret_20d", "IWM_SmallCap_vol_20d", "Core_CPI_zscore_60d", "MS_MorganStanley_ret_5d", "HangSeng_HK_ret_1d", "SPY_zscore_60d", "FedFunds_zscore_60d"], "is_new": true}, {"model_id": "new_h2_STRESS_RandomForest_N12_t7", "algo": "RandomForest", "regime": "STRESS", "horizon": 2, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWG_Germany_ret_20d", "SPY_zscore_60d", "EWC_Canada_zscore_60d", "EWQ_France_ret_20d", "AMT_AmericanTower_ret_1d", "Nikkei_Japan_vol_20d", "CLX_Clorox_vol_20d", "FedFunds_zscore_60d", "EWY_Korea_zscore_60d", "EWL_Switzerland_vol_20d"], "is_new": true}, {"model_id": "new_h2_STRESS_RandomForest_N15_t0", "algo": "RandomForest", "regime": "STRESS", "horizon": 2, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "QQQ_vol_20d", "TED_Spread_zscore_60d", "VRP_ma5", "EWH_HongKong_ret_5d", "JNJ_ret_1d", "AMT_AmericanTower_ret_1d", "LLY_zscore_60d", "vix_mean_abs_ret_5d", "heston_var_ev_h5", "EQR_Equity_ret_1d", "NEE_NextEra_ret_20d", "ASX_Australia_ret_5d", "ES_Evergy_ret_1d"], "is_new": true}, {"model_id": "new_h2_STRESS_RandomForest_N15_t1", "algo": "RandomForest", "regime": "STRESS", "horizon": 2, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "IYR_US_REIT2_zscore_60d", "Nikkei_Japan_zscore_60d", "PPL_PPL_ret_1d", "BLK_BlackRock_zscore_60d", "SBUX_ret_5d", "CI_Cigna_vol_20d", "heston_ev_h3", "TM_Telephone_vol_20d", "Core_PCE_zscore_60d", "EFFR_vol_20d", "WTI_Oil_FRED_zscore_60d", "EQR_Equity_ret_1d", "EWQ_France_zscore_60d"], "is_new": true}, {"model_id": "new_h2_STRESS_RandomForest_N15_t2", "algo": "RandomForest", "regime": "STRESS", "horizon": 2, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "US3M_Rate_zscore_60d", "CTAS_Cintas_vol_20d", "DAX_Germany_zscore_60d", "EWA_Australia_ret_1d", "SBUX_zscore_60d", "PG_ret_20d", "EWY_Korea_zscore_60d", "LUV_SouthwestAir_ret_5d", "WTI_Oil_FRED_zscore_60d", "heston_var_ev_h7", "VOD_Vodafone_zscore_60d", "T10Y2Y_Spread_ret_5d", "HangSeng_HK_ret_1d"], "is_new": true}, {"model_id": "new_h2_STRESS_RandomForest_N15_t3", "algo": "RandomForest", "regime": "STRESS", "horizon": 2, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "SJM_JM_Smucker_ret_1d", "NFCI_ret_5d", "SJM_JM_Smucker_ret_5d", "PAYX_Paychex_vol_20d", "AMD_ret_1d", "IYR_US_REIT2_zscore_60d", "SLB_Schlumberger_ret_1d", "SBUX_zscore_60d", "US6M_Rate_ret_20d", "BTI_BritishAmerican_ret_20d", "AXP_Amex_ret_20d", "XOM_ret_20d", "Michigan_Sentiment_ret_20d"], "is_new": true}, {"model_id": "new_h2_STRESS_RandomForest_N15_t4", "algo": "RandomForest", "regime": "STRESS", "horizon": 2, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "Nikkei_Japan_vol_20d", "GD_GeneralDynamics_zscore_60d", "AVB_AvalonBay_zscore_60d", "SCHW_Schwab_ret_5d", "ASX_Australia_vol_20d", "PCAR_PaccarInc_ret_5d", "TED_Spread_zscore_60d", "EWG_Germany_vol_20d", "TM_Telephone_ret_1d", "heston_var_ev_h7", "HangSeng_HK_ret_5d", "ORCL_zscore_60d", "SO_SouthernCo_ret_5d"], "is_new": true}, {"model_id": "new_h2_STRESS_RandomForest_N15_t5", "algo": "RandomForest", "regime": "STRESS", "horizon": 2, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "SPY_zscore_60d", "LMT_LockheedMartin_ret_1d", "PG_ret_20d", "EWJ_Japan_vol_20d", "Nikkei_Japan_zscore_60d", "IYM_BasicMaterials_ret_20d", "spx_momentum_3d", "HD_ret_20d", "3M_vol_20d", "HD_zscore_60d", "FedFunds_zscore_60d", "XLY_Disc_vol_20d", "DE_Deere_ret_5d"], "is_new": true}, {"model_id": "new_h2_STRESS_RandomForest_N15_t6", "algo": "RandomForest", "regime": "STRESS", "horizon": 2, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWA_Australia_ret_1d", "Core_PCE_zscore_60d", "PAYX_Paychex_zscore_60d", "US1Y_Rate_ret_20d", "SLB_Schlumberger_ret_1d", "EWS_Singapore_ret_5d", "spx_vol_5d", "MS_MorganStanley_ret_1d", "EXC_Exelon_ret_1d", "CI_Cigna_vol_20d", "Nikkei_Japan_vol_20d", "SCHW_Schwab_ret_5d", "SPY_zscore_60d"], "is_new": true}, {"model_id": "new_h2_STRESS_RandomForest_N15_t7", "algo": "RandomForest", "regime": "STRESS", "horizon": 2, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "AVB_AvalonBay_zscore_60d", "US1Y_Rate_ret_20d", "US30Y_Rate_ret_20d", "spx_momentum_3d", "US3Y_Rate_ret_5d", "INTC_ret_1d", "VOD_Vodafone_zscore_60d", "EWL_Switzerland_vol_20d", "AMZN_ret_5d", "EWG_Germany_ret_20d", "ES_Evergy_ret_1d", "XOM_ret_20d", "ASX_Australia_ret_5d"], "is_new": true}, {"model_id": "new_h2_STRESS_RandomForest_N20_t0", "algo": "RandomForest", "regime": "STRESS", "horizon": 2, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "JNJ_ret_1d", "EFFR_ret_1d", "NWL_Newell_ret_20d", "XLY_Disc_vol_20d", "XOM_ret_1d", "BA_ret_1d", "DAX_Germany_vol_20d", "LOW_Lowes_ret_5d", "EFFR_vol_20d", "EWG_Germany_vol_20d", "hmm_p_stress", "ASX_Australia_vol_20d", "DE_Deere_ret_5d", "XLF_Fin_vol_20d", "WTI_Oil_FRED_zscore_60d", "spx_momentum_3d", "M_Macys_vol_20d", "MSTR_Bitcoin3_ret_20d"], "is_new": true}, {"model_id": "new_h2_STRESS_RandomForest_N20_t1", "algo": "RandomForest", "regime": "STRESS", "horizon": 2, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWS_Singapore_ret_5d", "LOW_Lowes_ret_20d", "M_Macys_vol_20d", "LMT_LockheedMartin_vol_20d", "FedFunds_zscore_60d", "3M_ret_5d", "AORD_AUS_zscore_60d", "ORCL_zscore_60d", "VVIX_ret_20d", "heston_var_ev_h5", "VRP_ma5", "Core_PCE_zscore_60d", "GILD_Gilead_ret_20d", "BLK_BlackRock_zscore_60d", "EQR_Equity_ret_1d", "PG_ret_20d", "EWH_HongKong_ret_5d", "EMR_Emerson_ret_20d"], "is_new": true}, {"model_id": "new_h2_STRESS_RandomForest_N20_t2", "algo": "RandomForest", "regime": "STRESS", "horizon": 2, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWL_Switzerland_vol_20d", "CMCSA_ret_1d", "US1Y_Rate_ret_5d", "IWM_SmallCap_vol_20d", "3M_vol_20d", "AMT_AmericanTower_ret_1d", "HD_zscore_60d", "DAX_Germany_zscore_60d", "EWG_Germany_ret_20d", "CCI_CrownCastle_vol_20d", "vix_acceleration_1d", "TGT_Target_zscore_60d", "DHR_ret_1d", "ASX_Australia_vol_20d", "MSTR_Bitcoin3_ret_5d", "SPY_zscore_60d", "BDX_Becton_Dickinson_ret_20d", "HangSeng_HK_ret_5d"], "is_new": true}, {"model_id": "new_h2_STRESS_RandomForest_N20_t3", "algo": "RandomForest", "regime": "STRESS", "horizon": 2, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "BDX_Becton_Dickinson_ret_20d", "LLY_zscore_60d", "HangSeng_HK_ret_1d", "TED_Spread_zscore_60d", "HD_ret_5d", "PAYX_Paychex_vol_20d", "EWL_Switzerland_vol_20d", "SPY_zscore_60d", "LOW_Lowes_ret_20d", "PAYX_Paychex_ret_20d", "SBUX_vol_20d", "WTI_Oil_FRED_zscore_60d", "EWG_Germany_vol_20d", "VOD_Vodafone_zscore_60d", "TED_Spread_vol_20d", "DOW_Price_zscore_60d", "GE_ret_1d", "IYM_BasicMaterials_ret_20d"], "is_new": true}, {"model_id": "new_h2_STRESS_RandomForest_N20_t4", "algo": "RandomForest", "regime": "STRESS", "horizon": 2, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "MRK_Merck_zscore_60d", "CI_Cigna_vol_20d", "TM_Telephone_ret_1d", "EFFR_vol_20d", "SBUX_vol_20d", "DIS_vol_20d", "EWA_Australia_ret_1d", "SO_SouthernCo_ret_5d", "DHR_ret_1d", "ORCL_vol_20d", "GILD_Gilead_ret_20d", "DE_Deere_ret_5d", "VOD_Vodafone_zscore_60d", "LMT_LockheedMartin_vol_20d", "CPB_CampbellSoup_ret_5d", "SLB_Schlumberger_ret_1d", "JNJ_ret_1d", "ES_Evergy_ret_1d"], "is_new": true}, {"model_id": "new_h2_STRESS_RandomForest_N20_t5", "algo": "RandomForest", "regime": "STRESS", "horizon": 2, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "TED_Spread_zscore_60d", "TED_Spread_vol_20d", "spx_vol_5d", "CCI_CrownCastle_vol_20d", "Industrial_Production_zscore_60d", "VRP_ma5", "AVB_AvalonBay_zscore_60d", "EWS_Singapore_ret_5d", "BA_ret_1d", "SO_SouthernCo_ret_5d", "DAX_Germany_vol_20d", "US5Y_Rate_ret_5d", "NWL_Newell_ret_20d", "XLY_Disc_vol_20d", "IYM_BasicMaterials_ret_20d", "HangSeng_HK_ret_5d", "FedFunds_zscore_60d", "SJM_JM_Smucker_ret_5d"], "is_new": true}, {"model_id": "new_h2_STRESS_RandomForest_N20_t6", "algo": "RandomForest", "regime": "STRESS", "horizon": 2, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "AMD_ret_5d", "HUM_Humana_ret_5d", "SO_SouthernCo_ret_5d", "US1Y_Rate_ret_20d", "HD_ret_20d", "US7Y_Rate_ret_20d", "Core_CPI_zscore_60d", "IWM_SmallCap_vol_20d", "AORD_AUS_zscore_60d", "SLB_Schlumberger_ret_5d", "US3Y_Rate_ret_5d", "DE_Deere_vol_20d", "spx_abs_ret_max_5d", "EXC_Exelon_ret_1d", "HD_ret_5d", "NEE_NextEra_ret_20d", "HD_zscore_60d", "AVB_AvalonBay_zscore_60d"], "is_new": true}, {"model_id": "new_h2_STRESS_RandomForest_N20_t7", "algo": "RandomForest", "regime": "STRESS", "horizon": 2, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "NFCI_ret_5d", "VVIX_ret_20d", "LOW_Lowes_ret_5d", "LLY_zscore_60d", "LMT_LockheedMartin_vol_20d", "WTI_Oil_FRED_zscore_60d", "EWH_HongKong_ret_5d", "MO_AltriaMG_ret_1d", "IYR_US_REIT2_zscore_60d", "SLB_Schlumberger_ret_5d", "Brent_Oil_FRED_ret_5d", "BLK_BlackRock_zscore_60d", "SLB_Schlumberger_ret_1d", "DE_Deere_vol_20d", "spx_vol_5d", "EFFR_ret_1d", "PAYX_Paychex_vol_20d", "ORCL_zscore_60d"], "is_new": true}, {"model_id": "new_h2_STRESS_RandomForest_N25_t0", "algo": "RandomForest", "regime": "STRESS", "horizon": 2, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "XOM_ret_20d", "SJM_JM_Smucker_ret_1d", "SCHW_Schwab_ret_5d", "GE_ret_1d", "MRK_Merck_zscore_60d", "US5Y_Rate_ret_5d", "EQR_Equity_ret_1d", "MO_AltriaMG_ret_1d", "HD_ret_20d", "Core_PCE_zscore_60d", "DAX_Germany_zscore_60d", "EWL_Switzerland_vol_20d", "SO_SouthernCo_ret_5d", "PAYX_Paychex_vol_20d", "DIS_vol_20d", "HD_ret_5d", "VOD_Vodafone_zscore_60d", "EWY_Korea_ret_20d", "SBUX_ret_5d", "VVIX_ret_20d", "PPL_PPL_ret_1d", "ASX_Australia_vol_20d", "AMGN_Amgen_ret_1d"], "is_new": true}, {"model_id": "new_h2_STRESS_RandomForest_N25_t1", "algo": "RandomForest", "regime": "STRESS", "horizon": 2, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "LUV_SouthwestAir_ret_5d", "WTI_Oil_FRED_zscore_60d", "HD_ret_20d", "T_ret_1d", "IBEX_Spain_ret_20d", "US3M_Rate_zscore_60d", "NOC_Northrop_ret_20d", "PG_ret_20d", "FedFunds_zscore_60d", "CPB_CampbellSoup_zscore_60d", "ORCL_vol_20d", "Nikkei_Japan_vol_20d", "DE_Deere_ret_5d", "EWC_Canada_zscore_60d", "US3M_Rate_vol_20d", "JNJ_ret_1d", "NVDA_vol_20d", "SBUX_zscore_60d", "Core_PCE_zscore_60d", "EFFR_ret_1d", "IYM_BasicMaterials_ret_20d", "vix_mean_abs_ret_5d", "EWL_Switzerland_zscore_60d"], "is_new": true}, {"model_id": "new_h2_STRESS_RandomForest_N25_t2", "algo": "RandomForest", "regime": "STRESS", "horizon": 2, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "Nikkei_Japan_zscore_60d", "PAYX_Paychex_vol_20d", "FedFunds_zscore_60d", "XOM_ret_1d", "AMT_AmericanTower_ret_1d", "MSTR_Bitcoin3_ret_1d", "HangSeng_HK_ret_5d", "DAX_Germany_vol_20d", "EFFR_ret_1d", "heston_var_ev_h3", "SBUX_zscore_60d", "DIS_vol_20d", "SBUX_ret_5d", "MS_MorganStanley_zscore_60d", "LOW_Lowes_ret_5d", "EWC_Canada_zscore_60d", "BTI_BritishAmerican_ret_20d", "HD_ret_20d", "MO_AltriaMG_ret_1d", "CTAS_Cintas_vol_20d", "EWG_Germany_ret_20d", "GILD_Gilead_ret_20d", "CPB_CampbellSoup_vol_20d"], "is_new": true}, {"model_id": "new_h2_STRESS_RandomForest_N25_t3", "algo": "RandomForest", "regime": "STRESS", "horizon": 2, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "HD_ret_20d", "LUV_SouthwestAir_ret_5d", "Industrial_Production_zscore_60d", "US1Y_Rate_ret_5d", "EWQ_France_zscore_60d", "HD_zscore_60d", "LMT_LockheedMartin_vol_20d", "LOW_Lowes_ret_20d", "XOM_ret_1d", "spx_vol_5d", "vix_acceleration_1d", "ORCL_vol_20d", "ES_Evergy_ret_1d", "DAX_Germany_vol_20d", "NWL_Newell_ret_20d", "spx_momentum_3d", "EWJ_Japan_vol_20d", "EWG_Germany_vol_20d", "MRK_Merck_zscore_60d", "SBUX_ret_5d", "NOC_Northrop_ret_20d", "VRP_ma5", "SLB_Schlumberger_ret_5d"], "is_new": true}, {"model_id": "new_h2_STRESS_RandomForest_N25_t4", "algo": "RandomForest", "regime": "STRESS", "horizon": 2, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "US6M_Rate_ret_20d", "CMCSA_ret_1d", "AORD_AUS_zscore_60d", "SJM_JM_Smucker_ret_5d", "NOC_Northrop_ret_20d", "BA_ret_1d", "HUM_Humana_ret_5d", "EWG_Germany_ret_20d", "CCI_CrownCastle_vol_20d", "QQQ_vol_20d", "CLX_Clorox_vol_20d", "MO_AltriaMG_ret_1d", "IYM_BasicMaterials_ret_20d", "PPL_PPL_ret_1d", "US30Y_Rate_ret_20d", "ITT_ITTInc_ret_5d", "EFFR_vol_20d", "vix_acceleration_1d", "vix_mean_abs_ret_5d", "PCAR_PaccarInc_ret_5d", "US7Y_Rate_ret_20d", "EWM_Malaysia_zscore_60d", "EWM_Malaysia_ret_1d"], "is_new": true}, {"model_id": "new_h2_STRESS_RandomForest_N25_t5", "algo": "RandomForest", "regime": "STRESS", "horizon": 2, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "BTI_BritishAmerican_ret_5d", "NVDA_vol_20d", "ASX_Australia_vol_20d", "TED_Spread_vol_20d", "US5Y_Rate_ret_5d", "SPY_zscore_60d", "SLB_Schlumberger_ret_1d", "TXN_vol_20d", "LMT_LockheedMartin_ret_1d", "AMZN_ret_5d", "CLX_Clorox_vol_20d", "ASX_Australia_ret_5d", "MO_AltriaMG_ret_1d", "MRK_Merck_zscore_60d", "US1Y_Rate_ret_20d", "EWC_Canada_zscore_60d", "VVIX_ret_20d", "CCI_CrownCastle_vol_20d", "CMCSA_ret_1d", "CPB_CampbellSoup_zscore_60d", "XOM_ret_1d", "EWG_Germany_vol_20d", "EOG_EOGResources_vol_20d"], "is_new": true}, {"model_id": "new_h2_STRESS_RandomForest_N25_t6", "algo": "RandomForest", "regime": "STRESS", "horizon": 2, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "AXP_Amex_vol_20d", "ASX_Australia_ret_5d", "XLK_Tech_zscore_60d", "EWA_Australia_ret_1d", "EWQ_France_ret_20d", "EWL_Switzerland_vol_20d", "US3Y_Rate_ret_5d", "Nikkei_Japan_zscore_60d", "EWS_Singapore_ret_5d", "SJM_JM_Smucker_ret_1d", "VOD_Vodafone_zscore_60d", "AVB_AvalonBay_zscore_60d", "ES_Evergy_ret_1d", "3M_ret_5d", "spx_abs_ret_max_5d", "TXN_vol_20d", "US30Y_Rate_ret_20d", "PAYX_Paychex_vol_20d", "SPY_zscore_60d", "EOG_EOGResources_ret_5d", "BTI_BritishAmerican_ret_20d", "XLF_Fin_vol_20d", "TGT_Target_zscore_60d"], "is_new": true}, {"model_id": "new_h2_STRESS_RandomForest_N25_t7", "algo": "RandomForest", "regime": "STRESS", "horizon": 2, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "PAYX_Paychex_zscore_60d", "T10Y2Y_Spread_ret_5d", "CPB_CampbellSoup_ret_5d", "NFCI_ret_5d", "vix_mean_abs_ret_5d", "Nikkei_Japan_zscore_60d", "Michigan_Sentiment_ret_20d", "EWY_Korea_ret_20d", "GE_ret_1d", "DE_Deere_vol_20d", "AMD_ret_1d", "TXN_vol_20d", "EWM_Malaysia_zscore_60d", "HangSeng_HK_ret_1d", "SO_SouthernCo_ret_5d", "CMCSA_ret_1d", "ES_Evergy_ret_1d", "HUM_Humana_ret_5d", "XLF_Fin_vol_20d", "heston_var_ev_h7", "ORCL_vol_20d", "HD_zscore_60d", "SBUX_ret_5d"], "is_new": true}, {"model_id": "new_h2_STRESS_RandomForest_N30_t0", "algo": "RandomForest", "regime": "STRESS", "horizon": 2, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "GILD_Gilead_ret_20d", "AORD_AUS_zscore_60d", "EXC_Exelon_ret_1d", "PG_ret_20d", "heston_var_ev_h7", "US30Y_Rate_ret_20d", "EWA_Australia_zscore_60d", "ENB_EnbridgeInc_ret_1d", "SCHW_Schwab_ret_5d", "WTI_Oil_FRED_zscore_60d", "ORCL_vol_20d", "NEE_NextEra_ret_20d", "EWQ_France_zscore_60d", "PAYX_Paychex_ret_20d", "VVIX_ret_20d", "IWM_SmallCap_vol_20d", "SPY_zscore_60d", "CMCSA_ret_1d", "T_ret_1d", "QQQ_vol_20d", "XLY_Disc_vol_20d", "HangSeng_HK_ret_5d", "GE_ret_1d", "US3M_Rate_zscore_60d", "EOG_EOGResources_ret_5d", "IBEX_Spain_ret_20d", "MS_MorganStanley_ret_1d", "TED_Spread_vol_20d"], "is_new": true}, {"model_id": "new_h2_STRESS_RandomForest_N30_t1", "algo": "RandomForest", "regime": "STRESS", "horizon": 2, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "MSTR_Bitcoin3_ret_5d", "XLK_Tech_zscore_60d", "EWL_Switzerland_vol_20d", "US3M_Rate_zscore_60d", "XLV_Health_zscore_60d", "IYR_US_REIT2_zscore_60d", "EWY_Korea_ret_20d", "CMCSA_ret_1d", "IWM_SmallCap_vol_20d", "IYM_BasicMaterials_ret_20d", "ORCL_vol_20d", "Michigan_Sentiment_ret_20d", "IBEX_Spain_ret_20d", "DHR_vol_20d", "EWC_Canada_zscore_60d", "HD_ret_1d", "AMD_ret_1d", "EFFR_ret_1d", "MSTR_Bitcoin3_ret_1d", "US7Y_Rate_ret_20d", "XLF_Fin_vol_20d", "US1Y_Rate_ret_5d", "US3M_Rate_vol_20d", "WTI_Oil_FRED_zscore_60d", "GD_GeneralDynamics_zscore_60d", "MS_MorganStanley_ret_1d", "US3Y_Rate_ret_5d", "NVDA_vol_20d"], "is_new": true}, {"model_id": "new_h2_STRESS_RandomForest_N30_t2", "algo": "RandomForest", "regime": "STRESS", "horizon": 2, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "DHR_ret_1d", "US5Y_Rate_ret_5d", "DIS_vol_20d", "EWS_Singapore_ret_5d", "hmm_p_stress", "gjr_condvar_h1", "heston_var_ev_h7", "DE_Deere_vol_20d", "GD_GeneralDynamics_zscore_60d", "MSTR_Bitcoin3_ret_5d", "US30Y_Rate_ret_20d", "SBUX_zscore_60d", "vix_mean_abs_ret_5d", "ES_Evergy_ret_1d", "BA_ret_1d", "EWM_Malaysia_ret_1d", "PLD_Prologis_ret_5d", "EWH_HongKong_ret_5d", "EWM_Malaysia_vol_20d", "SBUX_vol_20d", "US1Y_Rate_ret_5d", "GE_ret_1d", "XLV_Health_zscore_60d", "MS_MorganStanley_ret_1d", "BTI_BritishAmerican_ret_20d", "EWG_Germany_vol_20d", "SO_SouthernCo_ret_5d", "NFCI_ret_5d"], "is_new": true}, {"model_id": "new_h2_STRESS_RandomForest_N30_t3", "algo": "RandomForest", "regime": "STRESS", "horizon": 2, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "SJM_JM_Smucker_ret_5d", "EWS_Singapore_ret_5d", "BA_ret_1d", "FedFunds_zscore_60d", "IYM_BasicMaterials_ret_20d", "IWM_SmallCap_vol_20d", "US7Y_Rate_ret_20d", "MSTR_Bitcoin3_ret_5d", "hmm_p_stress", "PAYX_Paychex_zscore_60d", "M_Macys_vol_20d", "MRK_Merck_zscore_60d", "PFE_ret_1d", "vix_mean_abs_ret_5d", "heston_var_ev_h5", "US30Y_Rate_ret_20d", "3M_ret_5d", "spx_vol_5d", "LMT_LockheedMartin_vol_20d", "Industrial_Production_zscore_60d", "BTI_BritishAmerican_ret_5d", "DE_Deere_ret_5d", "MS_MorganStanley_zscore_60d", "Retail_Sales_zscore_60d", "NWL_Newell_ret_20d", "EXC_Exelon_zscore_60d", "EXC_Exelon_ret_1d", "spx_momentum_3d"], "is_new": true}, {"model_id": "new_h2_STRESS_RandomForest_N30_t4", "algo": "RandomForest", "regime": "STRESS", "horizon": 2, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "PLD_Prologis_ret_5d", "ASX_Australia_vol_20d", "MO_AltriaMG_ret_1d", "NFCI_ret_5d", "CPB_CampbellSoup_zscore_60d", "EXC_Exelon_ret_1d", "ORCL_vol_20d", "LMT_LockheedMartin_vol_20d", "EWA_Australia_zscore_60d", "EWY_Korea_zscore_60d", "heston_ev_h3", "EQIX_Equinix_ret_5d", "AVB_AvalonBay_zscore_60d", "BLK_BlackRock_zscore_60d", "HD_zscore_60d", "DOW_Price_zscore_60d", "CLX_Clorox_vol_20d", "SLB_Schlumberger_ret_5d", "Nikkei_Japan_zscore_60d", "PAYX_Paychex_ret_20d", "gjr_condvar_h1", "PCAR_PaccarInc_ret_5d", "CPB_CampbellSoup_ret_5d", "spx_vol_5d", "DHR_ret_1d", "AXP_Amex_vol_20d", "EWY_Korea_ret_20d", "IWM_SmallCap_vol_20d"], "is_new": true}, {"model_id": "new_h2_STRESS_RandomForest_N30_t5", "algo": "RandomForest", "regime": "STRESS", "horizon": 2, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "JNJ_ret_1d", "NEE_NextEra_ret_20d", "MRK_Merck_zscore_60d", "XOM_ret_20d", "CPB_CampbellSoup_ret_5d", "EWQ_France_ret_20d", "EWM_Malaysia_ret_1d", "INTC_ret_5d", "T_ret_1d", "heston_var_ev_h5", "AXP_Amex_vol_20d", "TXN_vol_20d", "CLX_Clorox_vol_20d", "FedFunds_zscore_60d", "EWA_Australia_ret_1d", "ES_Evergy_ret_1d", "XLK_Tech_zscore_60d", "US3M_Rate_vol_20d", "SJM_JM_Smucker_ret_5d", "QQQ_vol_20d", "NWL_Newell_ret_20d", "NFCI_ret_5d", "EWG_Germany_vol_20d", "XLF_Fin_vol_20d", "SBUX_zscore_60d", "gjr_condvar_h1", "HD_ret_1d", "DAX_Germany_zscore_60d"], "is_new": true}, {"model_id": "new_h2_STRESS_RandomForest_N30_t6", "algo": "RandomForest", "regime": "STRESS", "horizon": 2, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "M_Macys_vol_20d", "AVB_AvalonBay_zscore_60d", "SLB_Schlumberger_ret_1d", "XLV_Health_zscore_60d", "Brent_Oil_FRED_ret_5d", "SBUX_zscore_60d", "AXP_Amex_ret_20d", "ASX_Australia_vol_20d", "LUV_SouthwestAir_ret_5d", "DOW_Price_zscore_60d", "DAX_Germany_zscore_60d", "XLB_Materials_zscore_60d", "CCI_CrownCastle_vol_20d", "TED_Spread_zscore_60d", "GD_GeneralDynamics_zscore_60d", "LMT_LockheedMartin_ret_1d", "PFE_ret_1d", "HUM_Humana_ret_5d", "US6M_Rate_ret_20d", "NEE_NextEra_ret_20d", "LOW_Lowes_ret_5d", "AMT_AmericanTower_ret_1d", "IYR_US_REIT2_zscore_60d", "DE_Deere_ret_5d", "QQQ_vol_20d", "AMGN_Amgen_ret_1d", "EQR_Equity_ret_1d", "XLK_Tech_zscore_60d"], "is_new": true}, {"model_id": "new_h2_STRESS_RandomForest_N30_t7", "algo": "RandomForest", "regime": "STRESS", "horizon": 2, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "HD_zscore_60d", "EWL_Switzerland_vol_20d", "INTC_ret_5d", "SJM_JM_Smucker_ret_5d", "EWA_Australia_zscore_60d", "AXP_Amex_ret_20d", "IBEX_Spain_ret_20d", "3M_vol_20d", "XOM_ret_20d", "SBUX_ret_5d", "Retail_Sales_zscore_60d", "LUV_SouthwestAir_ret_5d", "DAX_Germany_vol_20d", "EWL_Switzerland_zscore_60d", "AVB_AvalonBay_zscore_60d", "Nikkei_Japan_vol_20d", "Brent_Oil_FRED_ret_5d", "AMD_ret_1d", "VVIX_ret_20d", "EFFR_ret_1d", "DHR_ret_1d", "CPB_CampbellSoup_zscore_60d", "M_Macys_vol_20d", "T_ret_1d", "PCAR_PaccarInc_ret_5d", "HD_ret_5d", "Industrial_Production_zscore_60d", "EWG_Germany_ret_20d"], "is_new": true}, {"model_id": "new_h2_STRESS_LogisticRegression_N5_t0", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 2, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWQ_France_zscore_60d", "EFFR_ret_1d", "AXP_Amex_ret_20d"], "is_new": true}, {"model_id": "new_h2_STRESS_LogisticRegression_N5_t1", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 2, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "PAYX_Paychex_ret_20d", "HD_ret_20d", "EWY_Korea_ret_20d"], "is_new": true}, {"model_id": "new_h2_STRESS_LogisticRegression_N5_t2", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 2, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "NFCI_ret_5d", "LMT_LockheedMartin_ret_1d", "MS_MorganStanley_zscore_60d"], "is_new": true}, {"model_id": "new_h2_STRESS_LogisticRegression_N5_t3", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 2, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "M_Macys_vol_20d", "CLX_Clorox_vol_20d", "NFCI_ret_5d"], "is_new": true}, {"model_id": "new_h2_STRESS_LogisticRegression_N5_t4", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 2, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "DIS_vol_20d", "EWY_Korea_zscore_60d", "PCAR_PaccarInc_ret_5d"], "is_new": true}, {"model_id": "new_h2_STRESS_LogisticRegression_N5_t5", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 2, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "NVDA_vol_20d", "PCAR_PaccarInc_ret_5d", "DHR_ret_1d"], "is_new": true}, {"model_id": "new_h2_STRESS_LogisticRegression_N5_t6", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 2, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "BDX_Becton_Dickinson_ret_20d", "DHR_vol_20d", "PCAR_PaccarInc_ret_5d"], "is_new": true}, {"model_id": "new_h2_STRESS_LogisticRegression_N5_t7", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 2, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "3M_vol_20d", "LOW_Lowes_ret_20d", "ORCL_vol_20d"], "is_new": true}, {"model_id": "new_h2_STRESS_LogisticRegression_N8_t0", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 2, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "US6M_Rate_ret_20d", "3M_ret_5d", "INTC_ret_5d", "PG_ret_20d", "Brent_Oil_FRED_ret_5d", "Brent_Oil_FRED_ret_20d"], "is_new": true}, {"model_id": "new_h2_STRESS_LogisticRegression_N8_t1", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 2, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "PCAR_PaccarInc_ret_5d", "WTI_Oil_FRED_zscore_60d", "EWG_Germany_ret_20d", "EQR_Equity_ret_1d", "EWQ_France_zscore_60d", "LUV_SouthwestAir_ret_5d"], "is_new": true}, {"model_id": "new_h2_STRESS_LogisticRegression_N8_t2", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 2, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "gjr_condvar_h1", "MO_AltriaMG_ret_1d", "GE_ret_1d", "TED_Spread_zscore_60d", "HD_ret_20d", "US7Y_Rate_ret_20d"], "is_new": true}, {"model_id": "new_h2_STRESS_LogisticRegression_N8_t3", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 2, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "IYM_BasicMaterials_ret_20d", "QQQ_vol_20d", "BDX_Becton_Dickinson_ret_20d", "PAYX_Paychex_ret_20d", "US6M_Rate_ret_20d", "HUM_Humana_ret_5d"], "is_new": true}, {"model_id": "new_h2_STRESS_LogisticRegression_N8_t4", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 2, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "ASX_Australia_ret_5d", "DIS_vol_20d", "GE_ret_1d", "EQIX_Equinix_ret_5d", "MS_MorganStanley_zscore_60d", "FedFunds_zscore_60d"], "is_new": true}, {"model_id": "new_h2_STRESS_LogisticRegression_N8_t5", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 2, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "US3Y_Rate_ret_5d", "AMZN_ret_5d", "DHR_vol_20d", "ASX_Australia_ret_5d", "HangSeng_HK_ret_5d", "BTI_BritishAmerican_ret_5d"], "is_new": true}, {"model_id": "new_h2_STRESS_LogisticRegression_N8_t6", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 2, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "TXN_vol_20d", "SLB_Schlumberger_ret_1d", "ASX_Australia_ret_5d", "INTC_ret_1d", "CPB_CampbellSoup_ret_5d", "AXP_Amex_vol_20d"], "is_new": true}, {"model_id": "new_h2_STRESS_LogisticRegression_N8_t7", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 2, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "MS_MorganStanley_ret_5d", "LUV_SouthwestAir_ret_5d", "EWY_Korea_ret_20d", "EWM_Malaysia_ret_1d", "NOC_Northrop_ret_20d", "EWG_Germany_ret_20d"], "is_new": true}, {"model_id": "new_h2_STRESS_LogisticRegression_N10_t0", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 2, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "MO_AltriaMG_ret_1d", "HangSeng_HK_ret_5d", "EWA_Australia_zscore_60d", "PLD_Prologis_ret_5d", "BA_ret_1d", "3M_ret_5d", "US5Y_Rate_ret_5d", "TM_Telephone_vol_20d"], "is_new": true}, {"model_id": "new_h2_STRESS_LogisticRegression_N10_t1", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 2, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWM_Malaysia_zscore_60d", "PAYX_Paychex_zscore_60d", "PAYX_Paychex_vol_20d", "ASX_Australia_ret_5d", "heston_var_ev_h3", "heston_var_ev_h7", "DIS_vol_20d", "CPB_CampbellSoup_ret_20d"], "is_new": true}, {"model_id": "new_h2_STRESS_LogisticRegression_N10_t2", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 2, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "BLK_BlackRock_zscore_60d", "CTAS_Cintas_vol_20d", "ES_Evergy_ret_1d", "JNJ_ret_1d", "XOM_ret_1d", "EOG_EOGResources_ret_5d", "US1Y_Rate_ret_20d", "EWM_Malaysia_ret_1d"], "is_new": true}, {"model_id": "new_h2_STRESS_LogisticRegression_N10_t3", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 2, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "heston_var_ev_h5", "TM_Telephone_vol_20d", "MRK_Merck_zscore_60d", "MSTR_Bitcoin3_ret_20d", "TED_Spread_zscore_60d", "CPB_CampbellSoup_zscore_60d", "PCAR_PaccarInc_ret_5d", "MSTR_Bitcoin3_ret_1d"], "is_new": true}, {"model_id": "new_h2_STRESS_LogisticRegression_N10_t4", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 2, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "LMT_LockheedMartin_vol_20d", "XLB_Materials_zscore_60d", "CPB_CampbellSoup_ret_20d", "SCHW_Schwab_ret_5d", "EWQ_France_ret_20d", "US3M_Rate_vol_20d", "EWQ_France_zscore_60d", "AORD_AUS_zscore_60d"], "is_new": true}, {"model_id": "new_h2_STRESS_LogisticRegression_N10_t5", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 2, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "LOW_Lowes_ret_5d", "VOD_Vodafone_zscore_60d", "CI_Cigna_vol_20d", "MSTR_Bitcoin3_ret_1d", "XLF_Fin_vol_20d", "T_ret_1d", "EWA_Australia_ret_1d", "EWY_Korea_ret_20d"], "is_new": true}, {"model_id": "new_h2_STRESS_LogisticRegression_N10_t6", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 2, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "MSTR_Bitcoin3_ret_1d", "HD_ret_5d", "heston_var_ev_h5", "SJM_JM_Smucker_ret_1d", "AORD_AUS_zscore_60d", "SPY_zscore_60d", "XLB_Materials_zscore_60d", "EXC_Exelon_ret_1d"], "is_new": true}, {"model_id": "new_h2_STRESS_LogisticRegression_N10_t7", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 2, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "LMT_LockheedMartin_ret_1d", "gjr_condvar_h1", "HUM_Humana_ret_5d", "LMT_LockheedMartin_vol_20d", "DOW_Price_zscore_60d", "IYR_US_REIT2_zscore_60d", "US3Y_Rate_ret_5d", "EWG_Germany_ret_20d"], "is_new": true}, {"model_id": "new_h2_STRESS_LogisticRegression_N12_t0", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 2, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "MSTR_Bitcoin3_ret_20d", "MO_AltriaMG_ret_1d", "3M_vol_20d", "US5Y_Rate_ret_5d", "EXC_Exelon_zscore_60d", "MS_MorganStanley_ret_5d", "EQR_Equity_ret_1d", "AMD_ret_5d", "PPL_PPL_ret_1d", "TM_Telephone_ret_1d"], "is_new": true}, {"model_id": "new_h2_STRESS_LogisticRegression_N12_t1", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 2, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "spx_abs_ret_max_5d", "EQR_Equity_ret_1d", "PAYX_Paychex_zscore_60d", "CTAS_Cintas_vol_20d", "US6M_Rate_ret_20d", "spx_vol_5d", "EFFR_vol_20d", "NFCI_ret_5d", "AMZN_ret_5d", "DAX_Germany_zscore_60d"], "is_new": true}, {"model_id": "new_h2_STRESS_LogisticRegression_N12_t2", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 2, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "US30Y_Rate_ret_20d", "ORCL_vol_20d", "SO_SouthernCo_ret_5d", "AMT_AmericanTower_ret_1d", "Core_PCE_zscore_60d", "AMZN_ret_5d", "XLF_Fin_vol_20d", "EWL_Switzerland_zscore_60d", "3M_ret_5d", "VOD_Vodafone_zscore_60d"], "is_new": true}, {"model_id": "new_h2_STRESS_LogisticRegression_N12_t3", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 2, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "US1Y_Rate_ret_5d", "BTI_BritishAmerican_ret_5d", "spx_abs_ret_max_5d", "PLD_Prologis_ret_5d", "AXP_Amex_vol_20d", "EWC_Canada_zscore_60d", "DE_Deere_vol_20d", "SJM_JM_Smucker_ret_1d", "Michigan_Sentiment_ret_20d", "vix_acceleration_1d"], "is_new": true}, {"model_id": "new_h2_STRESS_LogisticRegression_N12_t4", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 2, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "PLD_Prologis_ret_5d", "SBUX_zscore_60d", "SCHW_Schwab_ret_5d", "AXP_Amex_vol_20d", "CI_Cigna_vol_20d", "HD_ret_1d", "vix_mean_abs_ret_5d", "VOD_Vodafone_zscore_60d", "PG_ret_20d", "DE_Deere_vol_20d"], "is_new": true}, {"model_id": "new_h2_STRESS_LogisticRegression_N12_t5", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 2, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "CPB_CampbellSoup_vol_20d", "US3M_Rate_vol_20d", "IYR_US_REIT2_zscore_60d", "Nikkei_Japan_vol_20d", "heston_var_ev_h7", "spx_abs_ret_max_5d", "WTI_Oil_FRED_zscore_60d", "M_Macys_vol_20d", "HangSeng_HK_vol_20d", "ORCL_vol_20d"], "is_new": true}, {"model_id": "new_h2_STRESS_LogisticRegression_N12_t6", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 2, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "heston_ev_h3", "INTC_ret_5d", "VVIX_ret_20d", "LOW_Lowes_ret_5d", "MS_MorganStanley_ret_1d", "US3M_Rate_vol_20d", "CTAS_Cintas_vol_20d", "VRP_ma5", "ENB_EnbridgeInc_ret_1d", "DOW_Price_zscore_60d"], "is_new": true}, {"model_id": "new_h2_STRESS_LogisticRegression_N12_t7", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 2, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "MO_AltriaMG_ret_1d", "M_Macys_vol_20d", "EWQ_France_zscore_60d", "US3M_Rate_zscore_60d", "spx_vol_5d", "PFE_ret_1d", "HD_ret_20d", "TED_Spread_zscore_60d", "vix_mean_abs_ret_5d", "XLK_Tech_zscore_60d"], "is_new": true}, {"model_id": "new_h2_STRESS_LogisticRegression_N15_t0", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 2, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "SCHW_Schwab_ret_5d", "LOW_Lowes_ret_20d", "CPB_CampbellSoup_zscore_60d", "NWL_Newell_ret_20d", "PLD_Prologis_ret_5d", "GD_GeneralDynamics_zscore_60d", "TED_Spread_vol_20d", "NOC_Northrop_ret_20d", "heston_var_ev_h3", "QQQ_vol_20d", "EWA_Australia_ret_1d", "LOW_Lowes_ret_5d", "TM_Telephone_vol_20d"], "is_new": true}, {"model_id": "new_h2_STRESS_LogisticRegression_N15_t1", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 2, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "SJM_JM_Smucker_ret_1d", "FedFunds_zscore_60d", "SLB_Schlumberger_ret_5d", "PAYX_Paychex_zscore_60d", "HD_ret_20d", "XLF_Fin_vol_20d", "EWQ_France_zscore_60d", "MRK_Merck_zscore_60d", "XLV_Health_zscore_60d", "XOM_ret_1d", "vix_acceleration_1d", "IYR_US_REIT2_zscore_60d", "spx_abs_ret_max_5d"], "is_new": true}, {"model_id": "new_h2_STRESS_LogisticRegression_N15_t2", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 2, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "MRK_Merck_zscore_60d", "MSTR_Bitcoin3_ret_5d", "XLB_Materials_zscore_60d", "US6M_Rate_ret_20d", "VRP_ma5", "PPL_PPL_ret_1d", "DAX_Germany_vol_20d", "EQIX_Equinix_ret_5d", "LMT_LockheedMartin_ret_1d", "DIS_vol_20d", "ASX_Australia_vol_20d", "CCI_CrownCastle_vol_20d", "XLK_Tech_zscore_60d"], "is_new": true}, {"model_id": "new_h2_STRESS_LogisticRegression_N15_t3", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 2, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "Nikkei_Japan_vol_20d", "heston_var_ev_h3", "US5Y_Rate_ret_5d", "DAX_Germany_vol_20d", "DOW_Price_zscore_60d", "MSTR_Bitcoin3_ret_20d", "SBUX_zscore_60d", "CTAS_Cintas_vol_20d", "CLX_Clorox_vol_20d", "US3M_Rate_zscore_60d", "NVDA_vol_20d", "US7Y_Rate_ret_20d", "EWA_Australia_ret_1d"], "is_new": true}, {"model_id": "new_h2_STRESS_LogisticRegression_N15_t4", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 2, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWJ_Japan_vol_20d", "XOM_ret_20d", "Brent_Oil_FRED_ret_5d", "EWS_Singapore_ret_5d", "US1Y_Rate_ret_5d", "Retail_Sales_zscore_60d", "EWL_Switzerland_vol_20d", "AVB_AvalonBay_zscore_60d", "ITT_ITTInc_ret_5d", "heston_var_ev_h5", "MS_MorganStanley_ret_5d", "TED_Spread_zscore_60d", "DE_Deere_ret_5d"], "is_new": true}, {"model_id": "new_h2_STRESS_LogisticRegression_N15_t5", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 2, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWQ_France_ret_20d", "IYM_BasicMaterials_ret_20d", "EQIX_Equinix_ret_5d", "NWL_Newell_ret_20d", "EOG_EOGResources_ret_5d", "BA_ret_1d", "DE_Deere_vol_20d", "WTI_Oil_FRED_zscore_60d", "GILD_Gilead_ret_20d", "T_ret_1d", "SBUX_zscore_60d", "EWY_Korea_zscore_60d", "Retail_Sales_zscore_60d"], "is_new": true}, {"model_id": "new_h2_STRESS_LogisticRegression_N15_t6", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 2, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "US1Y_Rate_ret_5d", "EFFR_vol_20d", "Michigan_Sentiment_ret_20d", "VRP_ma5", "vix_acceleration_1d", "VOD_Vodafone_zscore_60d", "EWC_Canada_zscore_60d", "HD_zscore_60d", "MO_AltriaMG_ret_1d", "DAX_Germany_vol_20d", "PAYX_Paychex_zscore_60d", "EWM_Malaysia_zscore_60d", "US3Y_Rate_ret_5d"], "is_new": true}, {"model_id": "new_h2_STRESS_LogisticRegression_N15_t7", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 2, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "CI_Cigna_vol_20d", "ORCL_zscore_60d", "heston_var_ev_h7", "PFE_ret_1d", "SLB_Schlumberger_ret_5d", "Nikkei_Japan_zscore_60d", "IYR_US_REIT2_zscore_60d", "AMD_ret_1d", "EWS_Singapore_ret_5d", "US3M_Rate_zscore_60d", "hmm_p_stress", "GILD_Gilead_ret_20d", "MRK_Merck_zscore_60d"], "is_new": true}, {"model_id": "new_h2_STRESS_LogisticRegression_N20_t0", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 2, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "SBUX_ret_5d", "spx_vol_5d", "SCHW_Schwab_ret_5d", "US3M_Rate_vol_20d", "CCI_CrownCastle_vol_20d", "PFE_ret_1d", "HangSeng_HK_ret_1d", "EWG_Germany_vol_20d", "Brent_Oil_FRED_ret_20d", "DAX_Germany_zscore_60d", "vix_mean_abs_ret_5d", "HangSeng_HK_ret_5d", "EWJ_Japan_vol_20d", "heston_var_ev_h5", "INTC_ret_5d", "LMT_LockheedMartin_ret_1d", "EWY_Korea_ret_20d", "SJM_JM_Smucker_ret_5d"], "is_new": true}, {"model_id": "new_h2_STRESS_LogisticRegression_N20_t1", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 2, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "SCHW_Schwab_ret_5d", "LMT_LockheedMartin_vol_20d", "ASX_Australia_ret_5d", "VOD_Vodafone_zscore_60d", "DHR_vol_20d", "PFE_ret_1d", "EFFR_vol_20d", "AXP_Amex_vol_20d", "XLY_Disc_vol_20d", "GE_ret_1d", "SO_SouthernCo_ret_5d", "EWA_Australia_ret_1d", "MSTR_Bitcoin3_ret_20d", "VRP_ma5", "MRK_Merck_zscore_60d", "PAYX_Paychex_zscore_60d", "heston_var_ev_h5", "NOC_Northrop_ret_20d"], "is_new": true}, {"model_id": "new_h2_STRESS_LogisticRegression_N20_t2", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 2, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "US5Y_Rate_ret_5d", "CPB_CampbellSoup_ret_5d", "NEE_NextEra_ret_20d", "TM_Telephone_ret_1d", "MSTR_Bitcoin3_ret_1d", "PFE_ret_1d", "3M_vol_20d", "T10Y2Y_Spread_ret_5d", "INTC_ret_1d", "IWM_SmallCap_vol_20d", "AMGN_Amgen_ret_1d", "M_Macys_vol_20d", "US3Y_Rate_ret_5d", "GE_ret_1d", "EWM_Malaysia_vol_20d", "US6M_Rate_ret_20d", "heston_var_ev_h5", "GD_GeneralDynamics_zscore_60d"], "is_new": true}, {"model_id": "new_h2_STRESS_LogisticRegression_N20_t3", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 2, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "HangSeng_HK_ret_1d", "DIS_vol_20d", "SBUX_vol_20d", "VRP_ma5", "QQQ_vol_20d", "EXC_Exelon_zscore_60d", "XOM_ret_1d", "EWA_Australia_zscore_60d", "LMT_LockheedMartin_ret_1d", "HD_ret_1d", "US7Y_Rate_ret_20d", "DOW_Price_zscore_60d", "Core_CPI_zscore_60d", "3M_ret_5d", "CPB_CampbellSoup_zscore_60d", "TED_Spread_vol_20d", "Brent_Oil_FRED_ret_20d", "SLB_Schlumberger_ret_1d"], "is_new": true}, {"model_id": "new_h2_STRESS_LogisticRegression_N20_t4", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 2, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EFFR_ret_1d", "XOM_ret_20d", "VOD_Vodafone_zscore_60d", "EWS_Singapore_ret_5d", "MO_AltriaMG_ret_1d", "PAYX_Paychex_vol_20d", "ITT_ITTInc_ret_5d", "EWH_HongKong_ret_5d", "TED_Spread_zscore_60d", "HangSeng_HK_vol_20d", "SJM_JM_Smucker_ret_5d", "heston_ev_h3", "Brent_Oil_FRED_ret_5d", "CPB_CampbellSoup_ret_5d", "CLX_Clorox_vol_20d", "IWM_SmallCap_vol_20d", "SJM_JM_Smucker_ret_1d", "ASX_Australia_vol_20d"], "is_new": true}, {"model_id": "new_h2_STRESS_LogisticRegression_N20_t5", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 2, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "MS_MorganStanley_zscore_60d", "BA_ret_1d", "heston_var_ev_h5", "CMCSA_ret_1d", "NFCI_ret_5d", "HD_ret_5d", "EMR_Emerson_ret_20d", "MSTR_Bitcoin3_ret_5d", "ES_Evergy_ret_1d", "HangSeng_HK_ret_5d", "IYR_US_REIT2_zscore_60d", "ITT_ITTInc_ret_5d", "HD_ret_20d", "NOC_Northrop_ret_20d", "3M_ret_5d", "EQIX_Equinix_ret_5d", "DAX_Germany_vol_20d", "EWM_Malaysia_zscore_60d"], "is_new": true}, {"model_id": "new_h2_STRESS_LogisticRegression_N20_t6", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 2, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "DHR_vol_20d", "spx_vol_5d", "EWY_Korea_ret_20d", "US3M_Rate_zscore_60d", "HD_ret_20d", "3M_vol_20d", "SBUX_zscore_60d", "CI_Cigna_vol_20d", "PAYX_Paychex_ret_20d", "vix_acceleration_1d", "HUM_Humana_ret_5d", "EWM_Malaysia_vol_20d", "GILD_Gilead_ret_20d", "MSTR_Bitcoin3_ret_5d", "BLK_BlackRock_zscore_60d", "XOM_ret_1d", "JNJ_ret_1d", "EXC_Exelon_zscore_60d"], "is_new": true}, {"model_id": "new_h2_STRESS_LogisticRegression_N20_t7", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 2, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "AXP_Amex_vol_20d", "EWY_Korea_ret_20d", "US3Y_Rate_ret_5d", "TED_Spread_vol_20d", "Brent_Oil_FRED_ret_5d", "BA_ret_1d", "PAYX_Paychex_ret_20d", "US5Y_Rate_ret_5d", "LOW_Lowes_ret_20d", "CPB_CampbellSoup_zscore_60d", "MS_MorganStanley_ret_5d", "EWA_Australia_ret_1d", "Industrial_Production_zscore_60d", "DE_Deere_vol_20d", "EXC_Exelon_zscore_60d", "gjr_condvar_h1", "CI_Cigna_vol_20d", "EQIX_Equinix_ret_5d"], "is_new": true}, {"model_id": "new_h2_STRESS_LogisticRegression_N25_t0", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 2, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "heston_ev_h3", "CPB_CampbellSoup_zscore_60d", "SBUX_vol_20d", "Industrial_Production_zscore_60d", "M_Macys_vol_20d", "BTI_BritishAmerican_ret_5d", "MSTR_Bitcoin3_ret_5d", "SO_SouthernCo_ret_5d", "AVB_AvalonBay_zscore_60d", "XLF_Fin_vol_20d", "spx_momentum_3d", "ORCL_vol_20d", "EWY_Korea_zscore_60d", "CI_Cigna_vol_20d", "TED_Spread_zscore_60d", "IYR_US_REIT2_zscore_60d", "GE_ret_1d", "DAX_Germany_vol_20d", "EWA_Australia_zscore_60d", "EWL_Switzerland_vol_20d", "EWC_Canada_zscore_60d", "XOM_ret_20d", "MS_MorganStanley_ret_5d"], "is_new": true}, {"model_id": "new_h2_STRESS_LogisticRegression_N25_t1", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 2, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "VVIX_ret_20d", "BTI_BritishAmerican_ret_5d", "IYM_BasicMaterials_ret_20d", "Nikkei_Japan_zscore_60d", "INTC_ret_5d", "US1Y_Rate_ret_20d", "Core_CPI_zscore_60d", "ORCL_zscore_60d", "XLK_Tech_zscore_60d", "CPB_CampbellSoup_ret_5d", "MRK_Merck_zscore_60d", "QQQ_vol_20d", "T_ret_1d", "EOG_EOGResources_ret_5d", "US3M_Rate_vol_20d", "BLK_BlackRock_zscore_60d", "spx_vol_5d", "EQIX_Equinix_ret_5d", "HangSeng_HK_ret_5d", "VRP_ma5", "XLB_Materials_zscore_60d", "XLY_Disc_vol_20d", "CI_Cigna_vol_20d"], "is_new": true}, {"model_id": "new_h2_STRESS_LogisticRegression_N25_t2", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 2, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EFFR_vol_20d", "US3Y_Rate_ret_5d", "US7Y_Rate_ret_20d", "XLF_Fin_vol_20d", "GE_ret_1d", "SJM_JM_Smucker_ret_5d", "CTAS_Cintas_vol_20d", "HD_zscore_60d", "heston_var_ev_h7", "HD_ret_20d", "3M_vol_20d", "CI_Cigna_vol_20d", "EWL_Switzerland_vol_20d", "SO_SouthernCo_ret_5d", "LOW_Lowes_ret_5d", "MS_MorganStanley_zscore_60d", "US5Y_Rate_ret_5d", "US6M_Rate_ret_20d", "EWA_Australia_ret_1d", "XLB_Materials_zscore_60d", "Core_PCE_zscore_60d", "heston_ev_h3", "EOG_EOGResources_ret_5d"], "is_new": true}, {"model_id": "new_h2_STRESS_LogisticRegression_N25_t3", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 2, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "Core_PCE_zscore_60d", "EMR_Emerson_ret_20d", "AORD_AUS_zscore_60d", "CPB_CampbellSoup_zscore_60d", "PAYX_Paychex_zscore_60d", "HangSeng_HK_ret_5d", "AVB_AvalonBay_zscore_60d", "DOW_Price_zscore_60d", "BA_ret_1d", "XLY_Disc_vol_20d", "SLB_Schlumberger_ret_1d", "hmm_p_stress", "HangSeng_HK_vol_20d", "ORCL_vol_20d", "SJM_JM_Smucker_ret_5d", "SO_SouthernCo_ret_5d", "T10Y2Y_Spread_ret_5d", "MS_MorganStanley_ret_5d", "Michigan_Sentiment_ret_20d", "LUV_SouthwestAir_ret_5d", "HangSeng_HK_ret_1d", "TXN_vol_20d", "Brent_Oil_FRED_ret_20d"], "is_new": true}, {"model_id": "new_h2_STRESS_LogisticRegression_N25_t4", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 2, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "WTI_Oil_FRED_zscore_60d", "DAX_Germany_vol_20d", "EWY_Korea_ret_20d", "MSTR_Bitcoin3_ret_5d", "PFE_ret_1d", "IWM_SmallCap_vol_20d", "XLF_Fin_vol_20d", "CPB_CampbellSoup_vol_20d", "NOC_Northrop_ret_20d", "IBEX_Spain_ret_20d", "GE_ret_1d", "US5Y_Rate_ret_5d", "vix_acceleration_1d", "INTC_ret_1d", "EWA_Australia_ret_1d", "DOW_Price_zscore_60d", "Core_CPI_zscore_60d", "US30Y_Rate_ret_20d", "AMZN_ret_5d", "CPB_CampbellSoup_ret_20d", "CPB_CampbellSoup_ret_5d", "XLV_Health_zscore_60d", "BLK_BlackRock_zscore_60d"], "is_new": true}, {"model_id": "new_h2_STRESS_LogisticRegression_N25_t5", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 2, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "CPB_CampbellSoup_vol_20d", "LOW_Lowes_ret_20d", "Michigan_Sentiment_ret_20d", "TGT_Target_zscore_60d", "EOG_EOGResources_ret_5d", "EWQ_France_ret_20d", "HD_zscore_60d", "EQR_Equity_ret_1d", "TXN_vol_20d", "EWY_Korea_zscore_60d", "AMD_ret_5d", "DAX_Germany_zscore_60d", "DIS_vol_20d", "heston_ev_h3", "MS_MorganStanley_zscore_60d", "EOG_EOGResources_vol_20d", "MS_MorganStanley_ret_5d", "Retail_Sales_zscore_60d", "CCI_CrownCastle_vol_20d", "EWS_Singapore_ret_5d", "EFFR_vol_20d", "AMD_ret_1d", "DHR_ret_1d"], "is_new": true}, {"model_id": "new_h2_STRESS_LogisticRegression_N25_t6", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 2, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWQ_France_zscore_60d", "TM_Telephone_ret_1d", "NWL_Newell_ret_20d", "ORCL_zscore_60d", "AVB_AvalonBay_zscore_60d", "PAYX_Paychex_vol_20d", "QQQ_vol_20d", "MSTR_Bitcoin3_ret_5d", "AORD_AUS_zscore_60d", "IBEX_Spain_ret_20d", "DE_Deere_vol_20d", "XLY_Disc_vol_20d", "EWC_Canada_zscore_60d", "VRP_ma5", "vix_acceleration_1d", "SCHW_Schwab_ret_5d", "Nikkei_Japan_vol_20d", "MS_MorganStanley_zscore_60d", "PPL_PPL_ret_1d", "HangSeng_HK_ret_5d", "BA_ret_1d", "TM_Telephone_vol_20d", "SJM_JM_Smucker_ret_1d"], "is_new": true}, {"model_id": "new_h2_STRESS_LogisticRegression_N25_t7", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 2, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "PAYX_Paychex_vol_20d", "hmm_p_stress", "Industrial_Production_zscore_60d", "EQIX_Equinix_ret_5d", "HD_zscore_60d", "BA_ret_1d", "AMT_AmericanTower_ret_1d", "EWQ_France_zscore_60d", "EWL_Switzerland_vol_20d", "ASX_Australia_vol_20d", "MO_AltriaMG_ret_1d", "Brent_Oil_FRED_ret_20d", "AMGN_Amgen_ret_1d", "WTI_Oil_FRED_zscore_60d", "SCHW_Schwab_ret_5d", "DIS_vol_20d", "HD_ret_20d", "PG_ret_20d", "DHR_ret_1d", "LLY_zscore_60d", "GILD_Gilead_ret_20d", "EQR_Equity_ret_1d", "Nikkei_Japan_zscore_60d"], "is_new": true}, {"model_id": "new_h2_STRESS_LogisticRegression_N30_t0", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 2, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "vix_acceleration_1d", "TGT_Target_zscore_60d", "VRP_ma5", "EWH_HongKong_ret_5d", "spx_vol_5d", "EQIX_Equinix_ret_5d", "SBUX_vol_20d", "ORCL_vol_20d", "EWM_Malaysia_zscore_60d", "US6M_Rate_ret_20d", "US1Y_Rate_ret_20d", "INTC_ret_5d", "ES_Evergy_ret_1d", "EWL_Switzerland_zscore_60d", "US1Y_Rate_ret_5d", "CPB_CampbellSoup_vol_20d", "gjr_condvar_h1", "EOG_EOGResources_ret_5d", "SLB_Schlumberger_ret_5d", "US5Y_Rate_ret_5d", "EWA_Australia_ret_1d", "MSTR_Bitcoin3_ret_20d", "EWA_Australia_zscore_60d", "ENB_EnbridgeInc_ret_1d", "PFE_ret_1d", "T_ret_1d", "Nikkei_Japan_vol_20d", "SPY_zscore_60d"], "is_new": true}, {"model_id": "new_h2_STRESS_LogisticRegression_N30_t1", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 2, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EQR_Equity_ret_1d", "hmm_p_stress", "EWM_Malaysia_zscore_60d", "LMT_LockheedMartin_vol_20d", "PCAR_PaccarInc_ret_5d", "vix_acceleration_1d", "spx_vol_5d", "CCI_CrownCastle_vol_20d", "T10Y2Y_Spread_ret_5d", "XLV_Health_zscore_60d", "XLB_Materials_zscore_60d", "MO_AltriaMG_ret_1d", "US6M_Rate_ret_20d", "EWQ_France_ret_20d", "Brent_Oil_FRED_ret_20d", "heston_var_ev_h5", "Nikkei_Japan_vol_20d", "XOM_ret_1d", "VRP_ma5", "ES_Evergy_ret_1d", "XLK_Tech_zscore_60d", "EWA_Australia_ret_1d", "INTC_ret_5d", "EWY_Korea_ret_20d", "IBEX_Spain_ret_20d", "PLD_Prologis_ret_5d", "BDX_Becton_Dickinson_ret_20d", "LLY_zscore_60d"], "is_new": true}, {"model_id": "new_h2_STRESS_LogisticRegression_N30_t2", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 2, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "Retail_Sales_zscore_60d", "MRK_Merck_zscore_60d", "3M_vol_20d", "US1Y_Rate_ret_5d", "US1Y_Rate_ret_20d", "HangSeng_HK_ret_1d", "US3M_Rate_vol_20d", "HD_ret_20d", "US5Y_Rate_ret_5d", "XLK_Tech_zscore_60d", "heston_ev_h3", "M_Macys_vol_20d", "EWC_Canada_zscore_60d", "LLY_zscore_60d", "PCAR_PaccarInc_ret_5d", "MSTR_Bitcoin3_ret_1d", "SLB_Schlumberger_ret_1d", "NOC_Northrop_ret_20d", "XLB_Materials_zscore_60d", "DAX_Germany_zscore_60d", "Brent_Oil_FRED_ret_20d", "EWQ_France_ret_20d", "TXN_vol_20d", "AMD_ret_1d", "IWM_SmallCap_vol_20d", "LOW_Lowes_ret_20d", "VRP_ma5", "EXC_Exelon_ret_1d"], "is_new": true}, {"model_id": "new_h2_STRESS_LogisticRegression_N30_t3", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 2, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EOG_EOGResources_ret_5d", "EMR_Emerson_ret_20d", "MO_AltriaMG_ret_1d", "US7Y_Rate_ret_20d", "IYR_US_REIT2_zscore_60d", "Industrial_Production_zscore_60d", "CPB_CampbellSoup_ret_5d", "EWC_Canada_zscore_60d", "EWA_Australia_ret_1d", "ASX_Australia_ret_5d", "IWM_SmallCap_vol_20d", "XLK_Tech_zscore_60d", "BTI_BritishAmerican_ret_5d", "LMT_LockheedMartin_ret_1d", "HUM_Humana_ret_5d", "CPB_CampbellSoup_ret_20d", "US6M_Rate_ret_20d", "NFCI_ret_5d", "EWH_HongKong_ret_5d", "LMT_LockheedMartin_vol_20d", "CTAS_Cintas_vol_20d", "heston_var_ev_h7", "AMZN_ret_5d", "XLB_Materials_zscore_60d", "SJM_JM_Smucker_ret_1d", "IBEX_Spain_ret_20d", "JNJ_ret_1d", "CPB_CampbellSoup_zscore_60d"], "is_new": true}, {"model_id": "new_h2_STRESS_LogisticRegression_N30_t4", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 2, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "M_Macys_vol_20d", "LOW_Lowes_ret_20d", "XLB_Materials_zscore_60d", "AMD_ret_1d", "IYR_US_REIT2_zscore_60d", "AORD_AUS_zscore_60d", "EWY_Korea_zscore_60d", "BTI_BritishAmerican_ret_20d", "SBUX_zscore_60d", "EWQ_France_ret_20d", "SJM_JM_Smucker_ret_1d", "DHR_vol_20d", "XOM_ret_20d", "heston_ev_h3", "EXC_Exelon_zscore_60d", "DE_Deere_vol_20d", "NVDA_vol_20d", "SBUX_vol_20d", "SJM_JM_Smucker_ret_5d", "PLD_Prologis_ret_5d", "AVB_AvalonBay_zscore_60d", "EWC_Canada_zscore_60d", "EOG_EOGResources_vol_20d", "EWL_Switzerland_vol_20d", "TXN_vol_20d", "CTAS_Cintas_vol_20d", "DHR_ret_1d", "SPY_zscore_60d"], "is_new": true}, {"model_id": "new_h2_STRESS_LogisticRegression_N30_t5", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 2, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "LMT_LockheedMartin_vol_20d", "HangSeng_HK_vol_20d", "PAYX_Paychex_ret_20d", "INTC_ret_5d", "DHR_ret_1d", "Core_PCE_zscore_60d", "Brent_Oil_FRED_ret_20d", "EMR_Emerson_ret_20d", "ORCL_zscore_60d", "spx_momentum_3d", "US6M_Rate_ret_20d", "NWL_Newell_ret_20d", "EWS_Singapore_ret_5d", "SJM_JM_Smucker_ret_1d", "EWL_Switzerland_vol_20d", "heston_var_ev_h7", "CLX_Clorox_vol_20d", "vix_acceleration_1d", "EWG_Germany_vol_20d", "EWQ_France_zscore_60d", "CPB_CampbellSoup_vol_20d", "AMGN_Amgen_ret_1d", "TGT_Target_zscore_60d", "SJM_JM_Smucker_ret_5d", "Michigan_Sentiment_ret_20d", "heston_var_ev_h5", "VVIX_ret_20d", "BA_ret_1d"], "is_new": true}, {"model_id": "new_h2_STRESS_LogisticRegression_N30_t6", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 2, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "AMZN_ret_5d", "EWQ_France_ret_20d", "TGT_Target_zscore_60d", "CPB_CampbellSoup_ret_20d", "SLB_Schlumberger_ret_5d", "AMGN_Amgen_ret_1d", "EWS_Singapore_ret_5d", "NOC_Northrop_ret_20d", "HangSeng_HK_ret_1d", "T_ret_1d", "MRK_Merck_zscore_60d", "CMCSA_ret_1d", "ASX_Australia_vol_20d", "AXP_Amex_ret_20d", "DE_Deere_ret_5d", "BTI_BritishAmerican_ret_5d", "SJM_JM_Smucker_ret_5d", "EFFR_ret_1d", "HangSeng_HK_ret_5d", "Brent_Oil_FRED_ret_5d", "EWY_Korea_zscore_60d", "vix_mean_abs_ret_5d", "TXN_vol_20d", "CLX_Clorox_vol_20d", "PAYX_Paychex_vol_20d", "XLY_Disc_vol_20d", "DE_Deere_vol_20d", "SPY_zscore_60d"], "is_new": true}, {"model_id": "new_h2_STRESS_LogisticRegression_N30_t7", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 2, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "heston_ev_h3", "T_ret_1d", "PPL_PPL_ret_1d", "TGT_Target_zscore_60d", "PAYX_Paychex_vol_20d", "heston_var_ev_h5", "gjr_condvar_h1", "EWA_Australia_zscore_60d", "ES_Evergy_ret_1d", "BTI_BritishAmerican_ret_20d", "HD_ret_20d", "US3M_Rate_zscore_60d", "XLF_Fin_vol_20d", "US3Y_Rate_ret_5d", "Retail_Sales_zscore_60d", "AMD_ret_1d", "EWH_HongKong_ret_5d", "DAX_Germany_zscore_60d", "AMT_AmericanTower_ret_1d", "HD_ret_1d", "EWA_Australia_ret_1d", "IBEX_Spain_ret_20d", "SCHW_Schwab_ret_5d", "EXC_Exelon_ret_1d", "EWJ_Japan_vol_20d", "EWG_Germany_vol_20d", "HangSeng_HK_vol_20d", "XOM_ret_20d"], "is_new": true}, {"model_id": "new_h2_GLOBAL_XGBoost_N5_t0", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 2, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "LOW_Lowes_ret_5d", "BTI_BritishAmerican_ret_20d", "AXP_Amex_ret_20d"], "is_new": true}, {"model_id": "new_h2_GLOBAL_XGBoost_N5_t1", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 2, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "BTI_BritishAmerican_ret_20d", "AMD_ret_5d", "gjr_condvar_h1"], "is_new": true}, {"model_id": "new_h2_GLOBAL_XGBoost_N5_t2", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 2, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWQ_France_ret_20d", "Industrial_Production_zscore_60d", "LUV_SouthwestAir_ret_5d"], "is_new": true}, {"model_id": "new_h2_GLOBAL_XGBoost_N5_t3", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 2, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "AXP_Amex_vol_20d", "PPL_PPL_ret_1d", "DIS_vol_20d"], "is_new": true}, {"model_id": "new_h2_GLOBAL_XGBoost_N5_t4", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 2, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "NWL_Newell_ret_20d", "PG_ret_20d", "DE_Deere_ret_5d"], "is_new": true}, {"model_id": "new_h2_GLOBAL_XGBoost_N5_t5", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 2, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "Brent_Oil_FRED_ret_5d", "T10Y2Y_Spread_ret_5d", "WTI_Oil_FRED_zscore_60d"], "is_new": true}, {"model_id": "new_h2_GLOBAL_XGBoost_N5_t6", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 2, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "NWL_Newell_ret_20d", "QQQ_vol_20d", "ORCL_zscore_60d"], "is_new": true}, {"model_id": "new_h2_GLOBAL_XGBoost_N5_t7", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 2, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "PG_ret_20d", "spx_momentum_3d", "Brent_Oil_FRED_ret_5d"], "is_new": true}, {"model_id": "new_h2_GLOBAL_XGBoost_N8_t0", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 2, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "XLV_Health_zscore_60d", "PCAR_PaccarInc_ret_5d", "heston_var_ev_h5", "heston_ev_h3", "EMR_Emerson_ret_20d", "AVB_AvalonBay_zscore_60d"], "is_new": true}, {"model_id": "new_h2_GLOBAL_XGBoost_N8_t1", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 2, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "GILD_Gilead_ret_20d", "LOW_Lowes_ret_5d", "Core_CPI_zscore_60d", "VOD_Vodafone_zscore_60d", "EWY_Korea_zscore_60d", "INTC_ret_5d"], "is_new": true}, {"model_id": "new_h2_GLOBAL_XGBoost_N8_t2", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 2, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWM_Malaysia_ret_1d", "HD_ret_1d", "Core_CPI_zscore_60d", "TM_Telephone_ret_1d", "US1Y_Rate_ret_5d", "AORD_AUS_zscore_60d"], "is_new": true}, {"model_id": "new_h2_GLOBAL_XGBoost_N8_t3", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 2, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWY_Korea_ret_20d", "XOM_ret_20d", "heston_ev_h3", "CPB_CampbellSoup_zscore_60d", "LMT_LockheedMartin_ret_1d", "SBUX_vol_20d"], "is_new": true}, {"model_id": "new_h2_GLOBAL_XGBoost_N8_t4", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 2, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "TM_Telephone_vol_20d", "heston_var_ev_h3", "IWM_SmallCap_vol_20d", "EWQ_France_zscore_60d", "spx_vol_5d", "MS_MorganStanley_ret_5d"], "is_new": true}, {"model_id": "new_h2_GLOBAL_XGBoost_N8_t5", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 2, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "HD_ret_1d", "HangSeng_HK_vol_20d", "WTI_Oil_FRED_zscore_60d", "EWA_Australia_ret_1d", "EWQ_France_ret_20d", "US7Y_Rate_ret_20d"], "is_new": true}, {"model_id": "new_h2_GLOBAL_XGBoost_N8_t6", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 2, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWY_Korea_zscore_60d", "AORD_AUS_zscore_60d", "NFCI_ret_5d", "EWM_Malaysia_vol_20d", "SPY_zscore_60d", "INTC_ret_1d"], "is_new": true}, {"model_id": "new_h2_GLOBAL_XGBoost_N8_t7", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 2, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "BA_ret_1d", "PLD_Prologis_ret_5d", "heston_ev_h3", "Brent_Oil_FRED_ret_5d", "LMT_LockheedMartin_vol_20d", "EWC_Canada_zscore_60d"], "is_new": true}, {"model_id": "new_h2_GLOBAL_XGBoost_N10_t0", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 2, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "LLY_zscore_60d", "LOW_Lowes_ret_20d", "hmm_p_stress", "LMT_LockheedMartin_ret_1d", "Industrial_Production_zscore_60d", "EWL_Switzerland_zscore_60d", "INTC_ret_5d", "GD_GeneralDynamics_zscore_60d"], "is_new": true}, {"model_id": "new_h2_GLOBAL_XGBoost_N10_t1", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 2, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "PCAR_PaccarInc_ret_5d", "PLD_Prologis_ret_5d", "SJM_JM_Smucker_ret_5d", "TED_Spread_vol_20d", "TM_Telephone_ret_1d", "VOD_Vodafone_zscore_60d", "ENB_EnbridgeInc_ret_1d", "heston_var_ev_h5"], "is_new": true}, {"model_id": "new_h2_GLOBAL_XGBoost_N10_t2", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 2, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "ASX_Australia_vol_20d", "PCAR_PaccarInc_ret_5d", "DHR_ret_1d", "vix_acceleration_1d", "AMD_ret_1d", "GILD_Gilead_ret_20d", "AMT_AmericanTower_ret_1d", "XLY_Disc_vol_20d"], "is_new": true}, {"model_id": "new_h2_GLOBAL_XGBoost_N10_t3", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 2, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EOG_EOGResources_vol_20d", "HUM_Humana_ret_5d", "Michigan_Sentiment_ret_20d", "CPB_CampbellSoup_zscore_60d", "PAYX_Paychex_ret_20d", "LUV_SouthwestAir_ret_5d", "vix_mean_abs_ret_5d", "SBUX_zscore_60d"], "is_new": true}, {"model_id": "new_h2_GLOBAL_XGBoost_N10_t4", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 2, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "AMT_AmericanTower_ret_1d", "IBEX_Spain_ret_20d", "US1Y_Rate_ret_20d", "Nikkei_Japan_vol_20d", "NEE_NextEra_ret_20d", "SBUX_ret_5d", "CCI_CrownCastle_vol_20d", "T10Y2Y_Spread_ret_5d"], "is_new": true}, {"model_id": "new_h2_GLOBAL_XGBoost_N10_t5", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 2, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "Nikkei_Japan_zscore_60d", "DAX_Germany_zscore_60d", "HD_zscore_60d", "HangSeng_HK_ret_1d", "SJM_JM_Smucker_ret_1d", "BLK_BlackRock_zscore_60d", "EXC_Exelon_zscore_60d", "US3M_Rate_vol_20d"], "is_new": true}, {"model_id": "new_h2_GLOBAL_XGBoost_N10_t6", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 2, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "Core_CPI_zscore_60d", "AMGN_Amgen_ret_1d", "Core_PCE_zscore_60d", "SJM_JM_Smucker_ret_5d", "PG_ret_20d", "PLD_Prologis_ret_5d", "AMD_ret_5d", "EWM_Malaysia_zscore_60d"], "is_new": true}, {"model_id": "new_h2_GLOBAL_XGBoost_N10_t7", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 2, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "AXP_Amex_vol_20d", "HD_ret_5d", "spx_momentum_3d", "HUM_Humana_ret_5d", "EWM_Malaysia_ret_1d", "AORD_AUS_zscore_60d", "PG_ret_20d", "EWS_Singapore_ret_5d"], "is_new": true}, {"model_id": "new_h2_GLOBAL_XGBoost_N12_t0", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 2, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "TED_Spread_zscore_60d", "US3M_Rate_vol_20d", "XLV_Health_zscore_60d", "SBUX_ret_5d", "spx_momentum_3d", "EXC_Exelon_ret_1d", "WTI_Oil_FRED_zscore_60d", "BLK_BlackRock_zscore_60d", "EWJ_Japan_vol_20d", "heston_var_ev_h3"], "is_new": true}, {"model_id": "new_h2_GLOBAL_XGBoost_N12_t1", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 2, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "PG_ret_20d", "PAYX_Paychex_ret_20d", "XOM_ret_1d", "VVIX_ret_20d", "SJM_JM_Smucker_ret_5d", "VRP_ma5", "US3Y_Rate_ret_5d", "DHR_vol_20d", "EXC_Exelon_zscore_60d", "Michigan_Sentiment_ret_20d"], "is_new": true}, {"model_id": "new_h2_GLOBAL_XGBoost_N12_t2", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 2, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "AMT_AmericanTower_ret_1d", "HD_ret_20d", "TED_Spread_zscore_60d", "DAX_Germany_vol_20d", "CPB_CampbellSoup_vol_20d", "EWA_Australia_zscore_60d", "BLK_BlackRock_zscore_60d", "CMCSA_ret_1d", "SLB_Schlumberger_ret_5d", "Nikkei_Japan_vol_20d"], "is_new": true}, {"model_id": "new_h2_GLOBAL_XGBoost_N12_t3", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 2, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "CI_Cigna_vol_20d", "FedFunds_zscore_60d", "LOW_Lowes_ret_20d", "MSTR_Bitcoin3_ret_20d", "MRK_Merck_zscore_60d", "CPB_CampbellSoup_ret_20d", "SLB_Schlumberger_ret_5d", "DAX_Germany_vol_20d", "ENB_EnbridgeInc_ret_1d", "PG_ret_20d"], "is_new": true}, {"model_id": "new_h2_GLOBAL_XGBoost_N12_t4", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 2, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "AMT_AmericanTower_ret_1d", "EWS_Singapore_ret_5d", "GE_ret_1d", "EWQ_France_zscore_60d", "PFE_ret_1d", "EWC_Canada_zscore_60d", "EFFR_ret_1d", "LMT_LockheedMartin_vol_20d", "MSTR_Bitcoin3_ret_20d", "PPL_PPL_ret_1d"], "is_new": true}, {"model_id": "new_h2_GLOBAL_XGBoost_N12_t5", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 2, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "MSTR_Bitcoin3_ret_20d", "DHR_ret_1d", "INTC_ret_1d", "CMCSA_ret_1d", "hmm_p_stress", "vix_mean_abs_ret_5d", "EFFR_ret_1d", "EWG_Germany_vol_20d", "DAX_Germany_vol_20d", "Brent_Oil_FRED_ret_20d"], "is_new": true}, {"model_id": "new_h2_GLOBAL_XGBoost_N12_t6", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 2, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "Core_CPI_zscore_60d", "VRP_ma5", "AMZN_ret_5d", "heston_var_ev_h3", "DHR_ret_1d", "AORD_AUS_zscore_60d", "MSTR_Bitcoin3_ret_20d", "AMD_ret_1d", "SBUX_zscore_60d", "CPB_CampbellSoup_vol_20d"], "is_new": true}, {"model_id": "new_h2_GLOBAL_XGBoost_N12_t7", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 2, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "ASX_Australia_ret_5d", "LOW_Lowes_ret_5d", "DAX_Germany_zscore_60d", "EQIX_Equinix_ret_5d", "Brent_Oil_FRED_ret_5d", "IBEX_Spain_ret_20d", "FedFunds_zscore_60d", "EMR_Emerson_ret_20d", "heston_var_ev_h7", "NWL_Newell_ret_20d"], "is_new": true}, {"model_id": "new_h2_GLOBAL_XGBoost_N15_t0", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 2, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "US30Y_Rate_ret_20d", "FedFunds_zscore_60d", "vix_mean_abs_ret_5d", "DE_Deere_vol_20d", "INTC_ret_1d", "NWL_Newell_ret_20d", "Core_PCE_zscore_60d", "DAX_Germany_vol_20d", "NEE_NextEra_ret_20d", "CPB_CampbellSoup_zscore_60d", "XOM_ret_1d", "US3Y_Rate_ret_5d", "XLK_Tech_zscore_60d"], "is_new": true}, {"model_id": "new_h2_GLOBAL_XGBoost_N15_t1", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 2, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "CI_Cigna_vol_20d", "TGT_Target_zscore_60d", "GILD_Gilead_ret_20d", "IYM_BasicMaterials_ret_20d", "INTC_ret_1d", "EWH_HongKong_ret_5d", "ORCL_zscore_60d", "INTC_ret_5d", "TM_Telephone_ret_1d", "EFFR_ret_1d", "M_Macys_vol_20d", "TED_Spread_zscore_60d", "DAX_Germany_zscore_60d"], "is_new": true}, {"model_id": "new_h2_GLOBAL_XGBoost_N15_t2", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 2, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "GE_ret_1d", "XLK_Tech_zscore_60d", "US6M_Rate_ret_20d", "spx_momentum_3d", "IBEX_Spain_ret_20d", "ASX_Australia_ret_5d", "EQR_Equity_ret_1d", "EWA_Australia_zscore_60d", "DE_Deere_vol_20d", "LLY_zscore_60d", "US1Y_Rate_ret_5d", "MO_AltriaMG_ret_1d", "VRP_ma5"], "is_new": true}, {"model_id": "new_h2_GLOBAL_XGBoost_N15_t3", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 2, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "vix_mean_abs_ret_5d", "XOM_ret_20d", "IYR_US_REIT2_zscore_60d", "heston_var_ev_h7", "EOG_EOGResources_ret_5d", "SLB_Schlumberger_ret_5d", "SPY_zscore_60d", "AORD_AUS_zscore_60d", "Industrial_Production_zscore_60d", "SCHW_Schwab_ret_5d", "INTC_ret_1d", "AXP_Amex_ret_20d", "SJM_JM_Smucker_ret_1d"], "is_new": true}, {"model_id": "new_h2_GLOBAL_XGBoost_N15_t4", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 2, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "LOW_Lowes_ret_5d", "EXC_Exelon_zscore_60d", "HUM_Humana_ret_5d", "DE_Deere_ret_5d", "LOW_Lowes_ret_20d", "EQR_Equity_ret_1d", "HD_ret_1d", "QQQ_vol_20d", "SCHW_Schwab_ret_5d", "SBUX_zscore_60d", "SBUX_ret_5d", "EWY_Korea_zscore_60d", "HD_ret_5d"], "is_new": true}, {"model_id": "new_h2_GLOBAL_XGBoost_N15_t5", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 2, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "ASX_Australia_ret_5d", "HangSeng_HK_vol_20d", "Nikkei_Japan_vol_20d", "HD_ret_20d", "BA_ret_1d", "AXP_Amex_vol_20d", "ORCL_vol_20d", "AMD_ret_1d", "MSTR_Bitcoin3_ret_5d", "US3M_Rate_zscore_60d", "TGT_Target_zscore_60d", "SLB_Schlumberger_ret_5d", "DE_Deere_vol_20d"], "is_new": true}, {"model_id": "new_h2_GLOBAL_XGBoost_N15_t6", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 2, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "vix_mean_abs_ret_5d", "PFE_ret_1d", "LUV_SouthwestAir_ret_5d", "EWH_HongKong_ret_5d", "SJM_JM_Smucker_ret_1d", "PG_ret_20d", "VRP_ma5", "EWM_Malaysia_vol_20d", "US30Y_Rate_ret_20d", "LMT_LockheedMartin_vol_20d", "hmm_p_stress", "HD_zscore_60d", "spx_abs_ret_max_5d"], "is_new": true}, {"model_id": "new_h2_GLOBAL_XGBoost_N15_t7", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 2, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "DAX_Germany_vol_20d", "AMT_AmericanTower_ret_1d", "EXC_Exelon_ret_1d", "AMZN_ret_5d", "PLD_Prologis_ret_5d", "EFFR_vol_20d", "LOW_Lowes_ret_5d", "EWY_Korea_zscore_60d", "LMT_LockheedMartin_vol_20d", "EWC_Canada_zscore_60d", "XLB_Materials_zscore_60d", "ORCL_vol_20d", "AXP_Amex_ret_20d"], "is_new": true}, {"model_id": "new_h2_GLOBAL_XGBoost_N20_t0", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 2, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "DHR_vol_20d", "gjr_condvar_h1", "EWY_Korea_zscore_60d", "PLD_Prologis_ret_5d", "AORD_AUS_zscore_60d", "ORCL_zscore_60d", "US3M_Rate_vol_20d", "MS_MorganStanley_zscore_60d", "DE_Deere_vol_20d", "LOW_Lowes_ret_5d", "vix_mean_abs_ret_5d", "CPB_CampbellSoup_vol_20d", "Retail_Sales_zscore_60d", "HUM_Humana_ret_5d", "EWG_Germany_ret_20d", "DAX_Germany_vol_20d", "MS_MorganStanley_ret_5d", "BDX_Becton_Dickinson_ret_20d"], "is_new": true}, {"model_id": "new_h2_GLOBAL_XGBoost_N20_t1", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 2, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "hmm_p_stress", "EWM_Malaysia_zscore_60d", "US7Y_Rate_ret_20d", "HangSeng_HK_ret_1d", "EWY_Korea_ret_20d", "heston_ev_h3", "VVIX_ret_20d", "SLB_Schlumberger_ret_5d", "TED_Spread_zscore_60d", "EXC_Exelon_zscore_60d", "GD_GeneralDynamics_zscore_60d", "EFFR_ret_1d", "PFE_ret_1d", "US1Y_Rate_ret_5d", "US3Y_Rate_ret_5d", "LOW_Lowes_ret_20d", "EWS_Singapore_ret_5d", "Nikkei_Japan_vol_20d"], "is_new": true}, {"model_id": "new_h2_GLOBAL_XGBoost_N20_t2", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 2, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "FedFunds_zscore_60d", "Industrial_Production_zscore_60d", "HangSeng_HK_vol_20d", "XLB_Materials_zscore_60d", "LMT_LockheedMartin_vol_20d", "XOM_ret_1d", "PCAR_PaccarInc_ret_5d", "SLB_Schlumberger_ret_1d", "Core_CPI_zscore_60d", "EOG_EOGResources_vol_20d", "PAYX_Paychex_vol_20d", "EWA_Australia_zscore_60d", "CPB_CampbellSoup_zscore_60d", "SCHW_Schwab_ret_5d", "HD_ret_5d", "spx_momentum_3d", "DHR_vol_20d", "ORCL_zscore_60d"], "is_new": true}, {"model_id": "new_h2_GLOBAL_XGBoost_N20_t3", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 2, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "DIS_vol_20d", "HD_ret_1d", "AMZN_ret_5d", "EMR_Emerson_ret_20d", "EWJ_Japan_vol_20d", "WTI_Oil_FRED_zscore_60d", "EWH_HongKong_ret_5d", "MS_MorganStanley_ret_5d", "vix_acceleration_1d", "DE_Deere_ret_5d", "QQQ_vol_20d", "MS_MorganStanley_ret_1d", "CPB_CampbellSoup_vol_20d", "PG_ret_20d", "NEE_NextEra_ret_20d", "GE_ret_1d", "AORD_AUS_zscore_60d", "GILD_Gilead_ret_20d"], "is_new": true}, {"model_id": "new_h2_GLOBAL_XGBoost_N20_t4", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 2, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "spx_vol_5d", "LUV_SouthwestAir_ret_5d", "CPB_CampbellSoup_ret_5d", "CCI_CrownCastle_vol_20d", "EMR_Emerson_ret_20d", "US1Y_Rate_ret_20d", "MSTR_Bitcoin3_ret_1d", "EWM_Malaysia_zscore_60d", "SLB_Schlumberger_ret_1d", "DAX_Germany_vol_20d", "AMGN_Amgen_ret_1d", "LMT_LockheedMartin_ret_1d", "3M_vol_20d", "DE_Deere_vol_20d", "DE_Deere_ret_5d", "TGT_Target_zscore_60d", "XLB_Materials_zscore_60d", "ASX_Australia_vol_20d"], "is_new": true}, {"model_id": "new_h2_GLOBAL_XGBoost_N20_t5", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 2, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EOG_EOGResources_ret_5d", "TED_Spread_vol_20d", "DE_Deere_vol_20d", "PAYX_Paychex_ret_20d", "NVDA_vol_20d", "SBUX_zscore_60d", "SBUX_ret_5d", "GE_ret_1d", "EQR_Equity_ret_1d", "EWG_Germany_vol_20d", "XOM_ret_1d", "EWH_HongKong_ret_5d", "INTC_ret_5d", "FedFunds_zscore_60d", "EWL_Switzerland_vol_20d", "EWM_Malaysia_vol_20d", "heston_var_ev_h3", "PCAR_PaccarInc_ret_5d"], "is_new": true}, {"model_id": "new_h2_GLOBAL_XGBoost_N20_t6", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 2, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "XLF_Fin_vol_20d", "NVDA_vol_20d", "US6M_Rate_ret_20d", "ES_Evergy_ret_1d", "CCI_CrownCastle_vol_20d", "LLY_zscore_60d", "Core_PCE_zscore_60d", "AVB_AvalonBay_zscore_60d", "AMD_ret_5d", "DHR_ret_1d", "MSTR_Bitcoin3_ret_20d", "3M_ret_5d", "SBUX_zscore_60d", "ORCL_zscore_60d", "ENB_EnbridgeInc_ret_1d", "BTI_BritishAmerican_ret_5d", "EXC_Exelon_ret_1d", "EWA_Australia_zscore_60d"], "is_new": true}, {"model_id": "new_h2_GLOBAL_XGBoost_N20_t7", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 2, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "JNJ_ret_1d", "EWL_Switzerland_vol_20d", "SJM_JM_Smucker_ret_1d", "BTI_BritishAmerican_ret_5d", "LOW_Lowes_ret_5d", "LOW_Lowes_ret_20d", "QQQ_vol_20d", "EWY_Korea_ret_20d", "CTAS_Cintas_vol_20d", "AMZN_ret_5d", "EQR_Equity_ret_1d", "3M_vol_20d", "US30Y_Rate_ret_20d", "US3M_Rate_zscore_60d", "EWA_Australia_zscore_60d", "BTI_BritishAmerican_ret_20d", "US1Y_Rate_ret_5d", "PCAR_PaccarInc_ret_5d"], "is_new": true}, {"model_id": "new_h2_GLOBAL_XGBoost_N25_t0", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 2, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "QQQ_vol_20d", "XOM_ret_1d", "EWH_HongKong_ret_5d", "EWM_Malaysia_zscore_60d", "PLD_Prologis_ret_5d", "PAYX_Paychex_zscore_60d", "EWM_Malaysia_vol_20d", "CPB_CampbellSoup_vol_20d", "DE_Deere_ret_5d", "XLK_Tech_zscore_60d", "vix_mean_abs_ret_5d", "EWA_Australia_zscore_60d", "GILD_Gilead_ret_20d", "CI_Cigna_vol_20d", "IBEX_Spain_ret_20d", "SPY_zscore_60d", "JNJ_ret_1d", "BA_ret_1d", "EFFR_vol_20d", "PCAR_PaccarInc_ret_5d", "MRK_Merck_zscore_60d", "AMZN_ret_5d", "TED_Spread_vol_20d"], "is_new": true}, {"model_id": "new_h2_GLOBAL_XGBoost_N25_t1", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 2, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "XLF_Fin_vol_20d", "BLK_BlackRock_zscore_60d", "PPL_PPL_ret_1d", "DOW_Price_zscore_60d", "EOG_EOGResources_vol_20d", "AORD_AUS_zscore_60d", "US30Y_Rate_ret_20d", "XOM_ret_20d", "Retail_Sales_zscore_60d", "CI_Cigna_vol_20d", "EFFR_vol_20d", "DAX_Germany_vol_20d", "T10Y2Y_Spread_ret_5d", "US7Y_Rate_ret_20d", "HD_ret_1d", "LLY_zscore_60d", "EQR_Equity_ret_1d", "EWS_Singapore_ret_5d", "VVIX_ret_20d", "ITT_ITTInc_ret_5d", "US3M_Rate_zscore_60d", "NFCI_ret_5d", "ASX_Australia_vol_20d"], "is_new": true}, {"model_id": "new_h2_GLOBAL_XGBoost_N25_t2", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 2, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "IWM_SmallCap_vol_20d", "XLY_Disc_vol_20d", "XOM_ret_1d", "SPY_zscore_60d", "PG_ret_20d", "LMT_LockheedMartin_ret_1d", "Core_CPI_zscore_60d", "PFE_ret_1d", "LOW_Lowes_ret_5d", "PAYX_Paychex_vol_20d", "vix_acceleration_1d", "AORD_AUS_zscore_60d", "EWC_Canada_zscore_60d", "ENB_EnbridgeInc_ret_1d", "spx_vol_5d", "AMD_ret_5d", "US1Y_Rate_ret_20d", "TED_Spread_vol_20d", "TM_Telephone_ret_1d", "BDX_Becton_Dickinson_ret_20d", "ORCL_vol_20d", "3M_ret_5d", "HD_ret_20d"], "is_new": true}, {"model_id": "new_h2_GLOBAL_XGBoost_N25_t3", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 2, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EOG_EOGResources_ret_5d", "HD_ret_1d", "DAX_Germany_vol_20d", "EWL_Switzerland_zscore_60d", "AMZN_ret_5d", "AXP_Amex_vol_20d", "PAYX_Paychex_ret_20d", "SO_SouthernCo_ret_5d", "XOM_ret_20d", "TXN_vol_20d", "NWL_Newell_ret_20d", "SCHW_Schwab_ret_5d", "LMT_LockheedMartin_ret_1d", "SPY_zscore_60d", "SBUX_vol_20d", "IWM_SmallCap_vol_20d", "JNJ_ret_1d", "TM_Telephone_vol_20d", "Industrial_Production_zscore_60d", "3M_vol_20d", "VOD_Vodafone_zscore_60d", "WTI_Oil_FRED_zscore_60d", "US5Y_Rate_ret_5d"], "is_new": true}, {"model_id": "new_h2_GLOBAL_XGBoost_N25_t4", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 2, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "HUM_Humana_ret_5d", "XLY_Disc_vol_20d", "ORCL_zscore_60d", "EQIX_Equinix_ret_5d", "EWY_Korea_ret_20d", "MS_MorganStanley_ret_5d", "NFCI_ret_5d", "XOM_ret_1d", "HD_zscore_60d", "hmm_p_stress", "DAX_Germany_zscore_60d", "PAYX_Paychex_vol_20d", "spx_momentum_3d", "SJM_JM_Smucker_ret_5d", "Nikkei_Japan_vol_20d", "Industrial_Production_zscore_60d", "CPB_CampbellSoup_vol_20d", "TM_Telephone_vol_20d", "DAX_Germany_vol_20d", "HD_ret_5d", "DE_Deere_vol_20d", "AVB_AvalonBay_zscore_60d", "HangSeng_HK_vol_20d"], "is_new": true}, {"model_id": "new_h2_GLOBAL_XGBoost_N25_t5", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 2, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EXC_Exelon_zscore_60d", "M_Macys_vol_20d", "DIS_vol_20d", "CPB_CampbellSoup_zscore_60d", "SBUX_vol_20d", "MSTR_Bitcoin3_ret_5d", "vix_mean_abs_ret_5d", "EWA_Australia_zscore_60d", "HUM_Humana_ret_5d", "XLB_Materials_zscore_60d", "EWG_Germany_ret_20d", "US3Y_Rate_ret_5d", "HD_ret_5d", "spx_momentum_3d", "NWL_Newell_ret_20d", "MSTR_Bitcoin3_ret_1d", "PAYX_Paychex_vol_20d", "EXC_Exelon_ret_1d", "ASX_Australia_ret_5d", "PAYX_Paychex_ret_20d", "US1Y_Rate_ret_5d", "US30Y_Rate_ret_20d", "NVDA_vol_20d"], "is_new": true}, {"model_id": "new_h2_GLOBAL_XGBoost_N25_t6", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 2, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "GE_ret_1d", "XLY_Disc_vol_20d", "Brent_Oil_FRED_ret_5d", "CLX_Clorox_vol_20d", "BA_ret_1d", "MS_MorganStanley_zscore_60d", "heston_var_ev_h7", "ORCL_vol_20d", "PFE_ret_1d", "DOW_Price_zscore_60d", "Nikkei_Japan_vol_20d", "HD_zscore_60d", "CMCSA_ret_1d", "SLB_Schlumberger_ret_1d", "PAYX_Paychex_vol_20d", "EWQ_France_ret_20d", "US6M_Rate_ret_20d", "PAYX_Paychex_zscore_60d", "TM_Telephone_ret_1d", "INTC_ret_5d", "EWL_Switzerland_vol_20d", "PCAR_PaccarInc_ret_5d", "QQQ_vol_20d"], "is_new": true}, {"model_id": "new_h2_GLOBAL_XGBoost_N25_t7", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 2, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "US5Y_Rate_ret_5d", "EWJ_Japan_vol_20d", "CTAS_Cintas_vol_20d", "Nikkei_Japan_zscore_60d", "GD_GeneralDynamics_zscore_60d", "LOW_Lowes_ret_20d", "VVIX_ret_20d", "T10Y2Y_Spread_ret_5d", "DIS_vol_20d", "EXC_Exelon_zscore_60d", "EWY_Korea_ret_20d", "ORCL_zscore_60d", "TXN_vol_20d", "AMD_ret_5d", "GE_ret_1d", "XOM_ret_20d", "3M_ret_5d", "HD_zscore_60d", "AVB_AvalonBay_zscore_60d", "SBUX_ret_5d", "VOD_Vodafone_zscore_60d", "EWG_Germany_ret_20d", "MS_MorganStanley_ret_1d"], "is_new": true}, {"model_id": "new_h2_GLOBAL_XGBoost_N30_t0", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 2, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "MS_MorganStanley_ret_5d", "US6M_Rate_ret_20d", "EQIX_Equinix_ret_5d", "Nikkei_Japan_zscore_60d", "VOD_Vodafone_zscore_60d", "DHR_ret_1d", "AXP_Amex_vol_20d", "EWL_Switzerland_zscore_60d", "EFFR_vol_20d", "PPL_PPL_ret_1d", "EWH_HongKong_ret_5d", "JNJ_ret_1d", "SJM_JM_Smucker_ret_1d", "Core_PCE_zscore_60d", "DHR_vol_20d", "LUV_SouthwestAir_ret_5d", "MS_MorganStanley_ret_1d", "SBUX_ret_5d", "ES_Evergy_ret_1d", "Brent_Oil_FRED_ret_20d", "EFFR_ret_1d", "EWY_Korea_ret_20d", "3M_vol_20d", "AMZN_ret_5d", "US1Y_Rate_ret_20d", "XOM_ret_20d", "PFE_ret_1d", "SLB_Schlumberger_ret_1d"], "is_new": true}, {"model_id": "new_h2_GLOBAL_XGBoost_N30_t1", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 2, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWM_Malaysia_zscore_60d", "DHR_vol_20d", "HangSeng_HK_vol_20d", "EQR_Equity_ret_1d", "JNJ_ret_1d", "MS_MorganStanley_ret_1d", "MSTR_Bitcoin3_ret_20d", "AMT_AmericanTower_ret_1d", "3M_vol_20d", "US6M_Rate_ret_20d", "TM_Telephone_vol_20d", "3M_ret_5d", "IYR_US_REIT2_zscore_60d", "SBUX_zscore_60d", "TM_Telephone_ret_1d", "EWJ_Japan_vol_20d", "NEE_NextEra_ret_20d", "AXP_Amex_ret_20d", "EWG_Germany_vol_20d", "AORD_AUS_zscore_60d", "EWG_Germany_ret_20d", "GD_GeneralDynamics_zscore_60d", "Brent_Oil_FRED_ret_20d", "EWS_Singapore_ret_5d", "Michigan_Sentiment_ret_20d", "Industrial_Production_zscore_60d", "EWA_Australia_ret_1d", "DAX_Germany_zscore_60d"], "is_new": true}, {"model_id": "new_h2_GLOBAL_XGBoost_N30_t2", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 2, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "XLV_Health_zscore_60d", "XLB_Materials_zscore_60d", "PAYX_Paychex_ret_20d", "ORCL_zscore_60d", "AORD_AUS_zscore_60d", "SJM_JM_Smucker_ret_1d", "AXP_Amex_ret_20d", "Nikkei_Japan_vol_20d", "heston_var_ev_h7", "AMZN_ret_5d", "hmm_p_stress", "GE_ret_1d", "MSTR_Bitcoin3_ret_1d", "CPB_CampbellSoup_ret_5d", "HD_ret_1d", "EWQ_France_zscore_60d", "EWY_Korea_ret_20d", "spx_vol_5d", "ASX_Australia_ret_5d", "BA_ret_1d", "vix_mean_abs_ret_5d", "CCI_CrownCastle_vol_20d", "SBUX_vol_20d", "IBEX_Spain_ret_20d", "HD_ret_20d", "EWC_Canada_zscore_60d", "MS_MorganStanley_ret_1d", "US30Y_Rate_ret_20d"], "is_new": true}, {"model_id": "new_h2_GLOBAL_XGBoost_N30_t3", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 2, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "SJM_JM_Smucker_ret_5d", "PFE_ret_1d", "ORCL_vol_20d", "ES_Evergy_ret_1d", "CCI_CrownCastle_vol_20d", "LOW_Lowes_ret_20d", "Retail_Sales_zscore_60d", "SBUX_zscore_60d", "HangSeng_HK_ret_5d", "3M_vol_20d", "IWM_SmallCap_vol_20d", "EXC_Exelon_ret_1d", "MRK_Merck_zscore_60d", "vix_mean_abs_ret_5d", "spx_vol_5d", "Nikkei_Japan_vol_20d", "DAX_Germany_vol_20d", "TED_Spread_vol_20d", "DOW_Price_zscore_60d", "HD_zscore_60d", "TM_Telephone_vol_20d", "SJM_JM_Smucker_ret_1d", "HangSeng_HK_vol_20d", "BTI_BritishAmerican_ret_5d", "GILD_Gilead_ret_20d", "EOG_EOGResources_ret_5d", "heston_var_ev_h5", "SLB_Schlumberger_ret_1d"], "is_new": true}, {"model_id": "new_h2_GLOBAL_XGBoost_N30_t4", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 2, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "CPB_CampbellSoup_zscore_60d", "XLB_Materials_zscore_60d", "SJM_JM_Smucker_ret_5d", "spx_vol_5d", "PPL_PPL_ret_1d", "EOG_EOGResources_vol_20d", "NVDA_vol_20d", "EWH_HongKong_ret_5d", "EXC_Exelon_zscore_60d", "US3Y_Rate_ret_5d", "hmm_p_stress", "CI_Cigna_vol_20d", "EMR_Emerson_ret_20d", "DE_Deere_vol_20d", "EWG_Germany_vol_20d", "EWL_Switzerland_zscore_60d", "US5Y_Rate_ret_5d", "US7Y_Rate_ret_20d", "TED_Spread_vol_20d", "EFFR_vol_20d", "DAX_Germany_vol_20d", "INTC_ret_1d", "US3M_Rate_vol_20d", "AMZN_ret_5d", "AMD_ret_5d", "PG_ret_20d", "VRP_ma5", "EWJ_Japan_vol_20d"], "is_new": true}, {"model_id": "new_h2_GLOBAL_XGBoost_N30_t5", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 2, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "SLB_Schlumberger_ret_5d", "EXC_Exelon_zscore_60d", "LMT_LockheedMartin_ret_1d", "NVDA_vol_20d", "PCAR_PaccarInc_ret_5d", "SBUX_ret_5d", "Brent_Oil_FRED_ret_20d", "LUV_SouthwestAir_ret_5d", "GD_GeneralDynamics_zscore_60d", "CTAS_Cintas_vol_20d", "PG_ret_20d", "MSTR_Bitcoin3_ret_20d", "MS_MorganStanley_ret_1d", "3M_vol_20d", "HUM_Humana_ret_5d", "INTC_ret_1d", "NOC_Northrop_ret_20d", "T_ret_1d", "VOD_Vodafone_zscore_60d", "EWM_Malaysia_zscore_60d", "US5Y_Rate_ret_5d", "CI_Cigna_vol_20d", "CPB_CampbellSoup_ret_5d", "M_Macys_vol_20d", "IBEX_Spain_ret_20d", "MO_AltriaMG_ret_1d", "ORCL_zscore_60d", "US6M_Rate_ret_20d"], "is_new": true}, {"model_id": "new_h2_GLOBAL_XGBoost_N30_t6", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 2, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "QQQ_vol_20d", "Nikkei_Japan_zscore_60d", "MSTR_Bitcoin3_ret_5d", "M_Macys_vol_20d", "EWY_Korea_zscore_60d", "SBUX_ret_5d", "MS_MorganStanley_ret_5d", "AXP_Amex_ret_20d", "US30Y_Rate_ret_20d", "LMT_LockheedMartin_vol_20d", "DAX_Germany_zscore_60d", "AMD_ret_1d", "SJM_JM_Smucker_ret_5d", "JNJ_ret_1d", "DHR_ret_1d", "VOD_Vodafone_zscore_60d", "ORCL_vol_20d", "IBEX_Spain_ret_20d", "IYM_BasicMaterials_ret_20d", "ITT_ITTInc_ret_5d", "ES_Evergy_ret_1d", "Brent_Oil_FRED_ret_20d", "SBUX_zscore_60d", "LUV_SouthwestAir_ret_5d", "BLK_BlackRock_zscore_60d", "HD_ret_5d", "TM_Telephone_ret_1d", "SPY_zscore_60d"], "is_new": true}, {"model_id": "new_h2_GLOBAL_XGBoost_N30_t7", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 2, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "PLD_Prologis_ret_5d", "LOW_Lowes_ret_5d", "HangSeng_HK_ret_1d", "GILD_Gilead_ret_20d", "DHR_vol_20d", "ENB_EnbridgeInc_ret_1d", "3M_vol_20d", "Nikkei_Japan_zscore_60d", "Brent_Oil_FRED_ret_5d", "US1Y_Rate_ret_20d", "vix_acceleration_1d", "AVB_AvalonBay_zscore_60d", "NOC_Northrop_ret_20d", "XLF_Fin_vol_20d", "EOG_EOGResources_ret_5d", "VVIX_ret_20d", "TM_Telephone_ret_1d", "vix_mean_abs_ret_5d", "EWM_Malaysia_zscore_60d", "TED_Spread_vol_20d", "heston_var_ev_h5", "spx_momentum_3d", "ASX_Australia_ret_5d", "BLK_BlackRock_zscore_60d", "MSTR_Bitcoin3_ret_20d", "Retail_Sales_zscore_60d", "DHR_ret_1d", "HangSeng_HK_vol_20d"], "is_new": true}, {"model_id": "new_h2_GLOBAL_LightGBM_N5_t0", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 2, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWS_Singapore_ret_5d", "EMR_Emerson_ret_20d", "CLX_Clorox_vol_20d"], "is_new": true}, {"model_id": "new_h2_GLOBAL_LightGBM_N5_t1", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 2, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "CLX_Clorox_vol_20d", "TXN_vol_20d", "EWA_Australia_ret_1d"], "is_new": true}, {"model_id": "new_h2_GLOBAL_LightGBM_N5_t2", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 2, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "CCI_CrownCastle_vol_20d", "CPB_CampbellSoup_zscore_60d", "CTAS_Cintas_vol_20d"], "is_new": true}, {"model_id": "new_h2_GLOBAL_LightGBM_N5_t3", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 2, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "LMT_LockheedMartin_vol_20d", "EWY_Korea_zscore_60d", "SJM_JM_Smucker_ret_5d"], "is_new": true}, {"model_id": "new_h2_GLOBAL_LightGBM_N5_t4", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 2, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "INTC_ret_5d", "MRK_Merck_zscore_60d", "DE_Deere_vol_20d"], "is_new": true}, {"model_id": "new_h2_GLOBAL_LightGBM_N5_t5", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 2, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "US3M_Rate_zscore_60d", "MS_MorganStanley_ret_5d", "CPB_CampbellSoup_vol_20d"], "is_new": true}, {"model_id": "new_h2_GLOBAL_LightGBM_N5_t6", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 2, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "NFCI_ret_5d", "XLF_Fin_vol_20d", "PFE_ret_1d"], "is_new": true}, {"model_id": "new_h2_GLOBAL_LightGBM_N5_t7", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 2, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "INTC_ret_5d", "Brent_Oil_FRED_ret_20d", "PAYX_Paychex_zscore_60d"], "is_new": true}, {"model_id": "new_h2_GLOBAL_LightGBM_N8_t0", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 2, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "spx_momentum_3d", "EWQ_France_zscore_60d", "IYM_BasicMaterials_ret_20d", "LMT_LockheedMartin_ret_1d", "MS_MorganStanley_zscore_60d", "NWL_Newell_ret_20d"], "is_new": true}, {"model_id": "new_h2_GLOBAL_LightGBM_N8_t1", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 2, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "TM_Telephone_vol_20d", "US5Y_Rate_ret_5d", "LMT_LockheedMartin_vol_20d", "TM_Telephone_ret_1d", "ORCL_vol_20d", "vix_acceleration_1d"], "is_new": true}, {"model_id": "new_h2_GLOBAL_LightGBM_N8_t2", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 2, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "IWM_SmallCap_vol_20d", "MS_MorganStanley_ret_1d", "AXP_Amex_ret_20d", "DAX_Germany_vol_20d", "WTI_Oil_FRED_zscore_60d", "XLV_Health_zscore_60d"], "is_new": true}, {"model_id": "new_h2_GLOBAL_LightGBM_N8_t3", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 2, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "FedFunds_zscore_60d", "NWL_Newell_ret_20d", "XLB_Materials_zscore_60d", "HangSeng_HK_vol_20d", "heston_ev_h3", "PPL_PPL_ret_1d"], "is_new": true}, {"model_id": "new_h2_GLOBAL_LightGBM_N8_t4", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 2, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "TM_Telephone_ret_1d", "HD_ret_20d", "DHR_ret_1d", "Nikkei_Japan_zscore_60d", "TED_Spread_vol_20d", "LOW_Lowes_ret_5d"], "is_new": true}, {"model_id": "new_h2_GLOBAL_LightGBM_N8_t5", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 2, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "ORCL_vol_20d", "SBUX_ret_5d", "US3M_Rate_vol_20d", "XLF_Fin_vol_20d", "AXP_Amex_vol_20d", "CMCSA_ret_1d"], "is_new": true}, {"model_id": "new_h2_GLOBAL_LightGBM_N8_t6", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 2, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "XLF_Fin_vol_20d", "BDX_Becton_Dickinson_ret_20d", "XLY_Disc_vol_20d", "INTC_ret_1d", "DAX_Germany_zscore_60d", "AMD_ret_1d"], "is_new": true}, {"model_id": "new_h2_GLOBAL_LightGBM_N8_t7", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 2, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "DAX_Germany_zscore_60d", "DAX_Germany_vol_20d", "LUV_SouthwestAir_ret_5d", "heston_var_ev_h3", "Brent_Oil_FRED_ret_5d", "US3Y_Rate_ret_5d"], "is_new": true}, {"model_id": "new_h2_GLOBAL_LightGBM_N10_t0", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 2, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EOG_EOGResources_ret_5d", "HangSeng_HK_vol_20d", "GE_ret_1d", "NOC_Northrop_ret_20d", "EQR_Equity_ret_1d", "DOW_Price_zscore_60d", "ASX_Australia_vol_20d", "CPB_CampbellSoup_ret_20d"], "is_new": true}, {"model_id": "new_h2_GLOBAL_LightGBM_N10_t1", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 2, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWL_Switzerland_vol_20d", "gjr_condvar_h1", "EWQ_France_ret_20d", "XLB_Materials_zscore_60d", "EWJ_Japan_vol_20d", "LMT_LockheedMartin_vol_20d", "ASX_Australia_vol_20d", "T10Y2Y_Spread_ret_5d"], "is_new": true}, {"model_id": "new_h2_GLOBAL_LightGBM_N10_t2", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 2, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "MSTR_Bitcoin3_ret_5d", "DAX_Germany_zscore_60d", "ITT_ITTInc_ret_5d", "EWQ_France_zscore_60d", "TM_Telephone_ret_1d", "CMCSA_ret_1d", "GILD_Gilead_ret_20d", "AMD_ret_5d"], "is_new": true}, {"model_id": "new_h2_GLOBAL_LightGBM_N10_t3", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 2, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "INTC_ret_1d", "VVIX_ret_20d", "TED_Spread_vol_20d", "HD_ret_20d", "BTI_BritishAmerican_ret_20d", "GILD_Gilead_ret_20d", "EFFR_vol_20d", "BLK_BlackRock_zscore_60d"], "is_new": true}, {"model_id": "new_h2_GLOBAL_LightGBM_N10_t4", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 2, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "LLY_zscore_60d", "AXP_Amex_vol_20d", "SBUX_vol_20d", "HD_ret_1d", "MO_AltriaMG_ret_1d", "AMD_ret_5d", "WTI_Oil_FRED_zscore_60d", "EWJ_Japan_vol_20d"], "is_new": true}, {"model_id": "new_h2_GLOBAL_LightGBM_N10_t5", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 2, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "US7Y_Rate_ret_20d", "heston_var_ev_h3", "US3M_Rate_vol_20d", "XLF_Fin_vol_20d", "PLD_Prologis_ret_5d", "AMZN_ret_5d", "DHR_ret_1d", "FedFunds_zscore_60d"], "is_new": true}, {"model_id": "new_h2_GLOBAL_LightGBM_N10_t6", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 2, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "CLX_Clorox_vol_20d", "AMZN_ret_5d", "XLY_Disc_vol_20d", "AMD_ret_5d", "PLD_Prologis_ret_5d", "SBUX_ret_5d", "CPB_CampbellSoup_ret_20d", "vix_mean_abs_ret_5d"], "is_new": true}, {"model_id": "new_h2_GLOBAL_LightGBM_N10_t7", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 2, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWQ_France_zscore_60d", "DE_Deere_ret_5d", "SBUX_zscore_60d", "QQQ_vol_20d", "gjr_condvar_h1", "heston_var_ev_h5", "EWL_Switzerland_zscore_60d", "ORCL_vol_20d"], "is_new": true}, {"model_id": "new_h2_GLOBAL_LightGBM_N12_t0", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 2, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "MS_MorganStanley_zscore_60d", "BTI_BritishAmerican_ret_20d", "ORCL_zscore_60d", "DAX_Germany_zscore_60d", "PAYX_Paychex_ret_20d", "LOW_Lowes_ret_20d", "QQQ_vol_20d", "EWM_Malaysia_ret_1d", "Industrial_Production_zscore_60d", "ITT_ITTInc_ret_5d"], "is_new": true}, {"model_id": "new_h2_GLOBAL_LightGBM_N12_t1", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 2, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "CPB_CampbellSoup_ret_20d", "EQR_Equity_ret_1d", "MS_MorganStanley_ret_1d", "NEE_NextEra_ret_20d", "Nikkei_Japan_vol_20d", "CPB_CampbellSoup_zscore_60d", "Industrial_Production_zscore_60d", "EWY_Korea_ret_20d", "US6M_Rate_ret_20d", "EWL_Switzerland_vol_20d"], "is_new": true}, {"model_id": "new_h2_GLOBAL_LightGBM_N12_t2", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 2, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "PG_ret_20d", "BA_ret_1d", "DHR_ret_1d", "BTI_BritishAmerican_ret_5d", "LLY_zscore_60d", "BLK_BlackRock_zscore_60d", "heston_var_ev_h5", "CPB_CampbellSoup_ret_5d", "EWH_HongKong_ret_5d", "EXC_Exelon_ret_1d"], "is_new": true}, {"model_id": "new_h2_GLOBAL_LightGBM_N12_t3", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 2, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EFFR_vol_20d", "PPL_PPL_ret_1d", "XLY_Disc_vol_20d", "TM_Telephone_vol_20d", "XLK_Tech_zscore_60d", "heston_var_ev_h5", "IYM_BasicMaterials_ret_20d", "SLB_Schlumberger_ret_1d", "EQR_Equity_ret_1d", "HD_ret_5d"], "is_new": true}, {"model_id": "new_h2_GLOBAL_LightGBM_N12_t4", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 2, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "US3M_Rate_vol_20d", "BTI_BritishAmerican_ret_5d", "DE_Deere_ret_5d", "XLB_Materials_zscore_60d", "AXP_Amex_ret_20d", "EWL_Switzerland_zscore_60d", "PAYX_Paychex_zscore_60d", "CTAS_Cintas_vol_20d", "WTI_Oil_FRED_zscore_60d", "DIS_vol_20d"], "is_new": true}, {"model_id": "new_h2_GLOBAL_LightGBM_N12_t5", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 2, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "INTC_ret_1d", "PG_ret_20d", "vix_acceleration_1d", "CLX_Clorox_vol_20d", "HD_ret_5d", "EQR_Equity_ret_1d", "CI_Cigna_vol_20d", "NVDA_vol_20d", "ES_Evergy_ret_1d", "EQIX_Equinix_ret_5d"], "is_new": true}, {"model_id": "new_h2_GLOBAL_LightGBM_N12_t6", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 2, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "AXP_Amex_ret_20d", "3M_ret_5d", "GILD_Gilead_ret_20d", "EWY_Korea_zscore_60d", "EWA_Australia_ret_1d", "MS_MorganStanley_ret_5d", "EWL_Switzerland_vol_20d", "GD_GeneralDynamics_zscore_60d", "HD_zscore_60d", "EWH_HongKong_ret_5d"], "is_new": true}, {"model_id": "new_h2_GLOBAL_LightGBM_N12_t7", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 2, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "DIS_vol_20d", "EFFR_vol_20d", "3M_vol_20d", "gjr_condvar_h1", "BDX_Becton_Dickinson_ret_20d", "PG_ret_20d", "DOW_Price_zscore_60d", "ENB_EnbridgeInc_ret_1d", "PAYX_Paychex_ret_20d", "MS_MorganStanley_zscore_60d"], "is_new": true}, {"model_id": "new_h2_GLOBAL_LightGBM_N15_t0", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 2, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "PG_ret_20d", "SLB_Schlumberger_ret_5d", "AXP_Amex_vol_20d", "XOM_ret_20d", "SLB_Schlumberger_ret_1d", "PAYX_Paychex_vol_20d", "TXN_vol_20d", "INTC_ret_5d", "vix_mean_abs_ret_5d", "SBUX_ret_5d", "CPB_CampbellSoup_zscore_60d", "BDX_Becton_Dickinson_ret_20d", "XLB_Materials_zscore_60d"], "is_new": true}, {"model_id": "new_h2_GLOBAL_LightGBM_N15_t1", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 2, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "AVB_AvalonBay_zscore_60d", "AMD_ret_5d", "AXP_Amex_ret_20d", "VOD_Vodafone_zscore_60d", "BLK_BlackRock_zscore_60d", "vix_acceleration_1d", "NWL_Newell_ret_20d", "EWJ_Japan_vol_20d", "heston_ev_h3", "DHR_ret_1d", "JNJ_ret_1d", "SO_SouthernCo_ret_5d", "heston_var_ev_h7"], "is_new": true}, {"model_id": "new_h2_GLOBAL_LightGBM_N15_t2", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 2, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "Michigan_Sentiment_ret_20d", "MO_AltriaMG_ret_1d", "US1Y_Rate_ret_20d", "NFCI_ret_5d", "Core_PCE_zscore_60d", "Retail_Sales_zscore_60d", "ENB_EnbridgeInc_ret_1d", "spx_momentum_3d", "IBEX_Spain_ret_20d", "PG_ret_20d", "spx_abs_ret_max_5d", "AORD_AUS_zscore_60d", "EWQ_France_zscore_60d"], "is_new": true}, {"model_id": "new_h2_GLOBAL_LightGBM_N15_t3", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 2, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWL_Switzerland_zscore_60d", "ES_Evergy_ret_1d", "HD_ret_5d", "MSTR_Bitcoin3_ret_1d", "LMT_LockheedMartin_vol_20d", "hmm_p_stress", "vix_mean_abs_ret_5d", "NFCI_ret_5d", "TED_Spread_zscore_60d", "HD_ret_20d", "BTI_BritishAmerican_ret_20d", "IYR_US_REIT2_zscore_60d", "HUM_Humana_ret_5d"], "is_new": true}, {"model_id": "new_h2_GLOBAL_LightGBM_N15_t4", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 2, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "Core_PCE_zscore_60d", "PLD_Prologis_ret_5d", "INTC_ret_1d", "SCHW_Schwab_ret_5d", "EWG_Germany_vol_20d", "BTI_BritishAmerican_ret_5d", "SO_SouthernCo_ret_5d", "LUV_SouthwestAir_ret_5d", "CPB_CampbellSoup_zscore_60d", "SJM_JM_Smucker_ret_1d", "MS_MorganStanley_zscore_60d", "US3M_Rate_zscore_60d", "MSTR_Bitcoin3_ret_5d"], "is_new": true}, {"model_id": "new_h2_GLOBAL_LightGBM_N15_t5", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 2, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "NFCI_ret_5d", "AMD_ret_5d", "AVB_AvalonBay_zscore_60d", "heston_var_ev_h3", "EWM_Malaysia_vol_20d", "EWL_Switzerland_zscore_60d", "PAYX_Paychex_vol_20d", "INTC_ret_1d", "EFFR_ret_1d", "EQR_Equity_ret_1d", "US3M_Rate_zscore_60d", "MS_MorganStanley_zscore_60d", "GE_ret_1d"], "is_new": true}, {"model_id": "new_h2_GLOBAL_LightGBM_N15_t6", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 2, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "AMD_ret_1d", "US3M_Rate_vol_20d", "EFFR_vol_20d", "ES_Evergy_ret_1d", "DAX_Germany_zscore_60d", "IWM_SmallCap_vol_20d", "CMCSA_ret_1d", "FedFunds_zscore_60d", "Industrial_Production_zscore_60d", "LMT_LockheedMartin_vol_20d", "Retail_Sales_zscore_60d", "VOD_Vodafone_zscore_60d", "heston_ev_h3"], "is_new": true}, {"model_id": "new_h2_GLOBAL_LightGBM_N15_t7", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 2, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWM_Malaysia_vol_20d", "SBUX_vol_20d", "MS_MorganStanley_zscore_60d", "NFCI_ret_5d", "AXP_Amex_ret_20d", "GE_ret_1d", "BTI_BritishAmerican_ret_5d", "HD_ret_20d", "DHR_vol_20d", "Nikkei_Japan_vol_20d", "HangSeng_HK_vol_20d", "EWJ_Japan_vol_20d", "BDX_Becton_Dickinson_ret_20d"], "is_new": true}, {"model_id": "new_h2_GLOBAL_LightGBM_N20_t0", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 2, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWL_Switzerland_zscore_60d", "SBUX_vol_20d", "EFFR_ret_1d", "EQIX_Equinix_ret_5d", "VRP_ma5", "US30Y_Rate_ret_20d", "DHR_vol_20d", "DAX_Germany_vol_20d", "AMGN_Amgen_ret_1d", "XLY_Disc_vol_20d", "EWG_Germany_ret_20d", "LOW_Lowes_ret_5d", "heston_var_ev_h3", "DOW_Price_zscore_60d", "IBEX_Spain_ret_20d", "vix_mean_abs_ret_5d", "EWA_Australia_ret_1d", "XLF_Fin_vol_20d"], "is_new": true}, {"model_id": "new_h2_GLOBAL_LightGBM_N20_t1", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 2, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "CLX_Clorox_vol_20d", "vix_acceleration_1d", "HangSeng_HK_vol_20d", "US3M_Rate_zscore_60d", "MS_MorganStanley_ret_5d", "DE_Deere_vol_20d", "US1Y_Rate_ret_5d", "EXC_Exelon_zscore_60d", "AMZN_ret_5d", "PFE_ret_1d", "Nikkei_Japan_zscore_60d", "heston_var_ev_h7", "VRP_ma5", "IYM_BasicMaterials_ret_20d", "EWC_Canada_zscore_60d", "XLB_Materials_zscore_60d", "BA_ret_1d", "TXN_vol_20d"], "is_new": true}, {"model_id": "new_h2_GLOBAL_LightGBM_N20_t2", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 2, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "Michigan_Sentiment_ret_20d", "PFE_ret_1d", "XOM_ret_1d", "AMZN_ret_5d", "AXP_Amex_vol_20d", "ORCL_zscore_60d", "SLB_Schlumberger_ret_1d", "EFFR_ret_1d", "IBEX_Spain_ret_20d", "HangSeng_HK_ret_5d", "M_Macys_vol_20d", "Nikkei_Japan_zscore_60d", "MS_MorganStanley_ret_5d", "MSTR_Bitcoin3_ret_1d", "XLF_Fin_vol_20d", "US30Y_Rate_ret_20d", "EWL_Switzerland_zscore_60d", "3M_vol_20d"], "is_new": true}, {"model_id": "new_h2_GLOBAL_LightGBM_N20_t3", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 2, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "SLB_Schlumberger_ret_1d", "EFFR_vol_20d", "LOW_Lowes_ret_20d", "AMGN_Amgen_ret_1d", "TED_Spread_zscore_60d", "IWM_SmallCap_vol_20d", "IYR_US_REIT2_zscore_60d", "US6M_Rate_ret_20d", "EQR_Equity_ret_1d", "heston_ev_h3", "Nikkei_Japan_vol_20d", "ENB_EnbridgeInc_ret_1d", "US3Y_Rate_ret_5d", "Retail_Sales_zscore_60d", "ES_Evergy_ret_1d", "MSTR_Bitcoin3_ret_1d", "MSTR_Bitcoin3_ret_20d", "US3M_Rate_vol_20d"], "is_new": true}, {"model_id": "new_h2_GLOBAL_LightGBM_N20_t4", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 2, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "PAYX_Paychex_vol_20d", "EFFR_vol_20d", "SBUX_vol_20d", "EWS_Singapore_ret_5d", "gjr_condvar_h1", "HangSeng_HK_ret_5d", "NFCI_ret_5d", "TM_Telephone_vol_20d", "XLF_Fin_vol_20d", "ENB_EnbridgeInc_ret_1d", "QQQ_vol_20d", "BDX_Becton_Dickinson_ret_20d", "GE_ret_1d", "PLD_Prologis_ret_5d", "EQIX_Equinix_ret_5d", "EWA_Australia_zscore_60d", "EWQ_France_zscore_60d", "EWY_Korea_zscore_60d"], "is_new": true}, {"model_id": "new_h2_GLOBAL_LightGBM_N20_t5", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 2, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "MS_MorganStanley_zscore_60d", "CPB_CampbellSoup_vol_20d", "AXP_Amex_ret_20d", "LMT_LockheedMartin_vol_20d", "CPB_CampbellSoup_ret_5d", "HangSeng_HK_ret_1d", "WTI_Oil_FRED_zscore_60d", "MSTR_Bitcoin3_ret_5d", "CLX_Clorox_vol_20d", "TXN_vol_20d", "AXP_Amex_vol_20d", "MSTR_Bitcoin3_ret_1d", "SO_SouthernCo_ret_5d", "EOG_EOGResources_ret_5d", "SJM_JM_Smucker_ret_5d", "T_ret_1d", "T10Y2Y_Spread_ret_5d", "PPL_PPL_ret_1d"], "is_new": true}, {"model_id": "new_h2_GLOBAL_LightGBM_N20_t6", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 2, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "XLB_Materials_zscore_60d", "EXC_Exelon_ret_1d", "AMGN_Amgen_ret_1d", "EWS_Singapore_ret_5d", "hmm_p_stress", "XOM_ret_1d", "LOW_Lowes_ret_5d", "CI_Cigna_vol_20d", "T10Y2Y_Spread_ret_5d", "Nikkei_Japan_vol_20d", "ITT_ITTInc_ret_5d", "NOC_Northrop_ret_20d", "CLX_Clorox_vol_20d", "EWM_Malaysia_vol_20d", "vix_acceleration_1d", "HangSeng_HK_ret_1d", "GE_ret_1d", "AXP_Amex_vol_20d"], "is_new": true}, {"model_id": "new_h2_GLOBAL_LightGBM_N20_t7", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 2, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "TXN_vol_20d", "EWY_Korea_ret_20d", "AORD_AUS_zscore_60d", "DAX_Germany_vol_20d", "MS_MorganStanley_zscore_60d", "US6M_Rate_ret_20d", "EWA_Australia_ret_1d", "GD_GeneralDynamics_zscore_60d", "PPL_PPL_ret_1d", "NVDA_vol_20d", "US7Y_Rate_ret_20d", "BTI_BritishAmerican_ret_5d", "DE_Deere_vol_20d", "heston_var_ev_h7", "EWG_Germany_vol_20d", "SCHW_Schwab_ret_5d", "XLB_Materials_zscore_60d", "gjr_condvar_h1"], "is_new": true}, {"model_id": "new_h2_GLOBAL_LightGBM_N25_t0", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 2, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EFFR_vol_20d", "AORD_AUS_zscore_60d", "Brent_Oil_FRED_ret_5d", "MSTR_Bitcoin3_ret_5d", "EWL_Switzerland_vol_20d", "CPB_CampbellSoup_ret_20d", "SJM_JM_Smucker_ret_5d", "LUV_SouthwestAir_ret_5d", "EQR_Equity_ret_1d", "vix_acceleration_1d", "3M_vol_20d", "XOM_ret_20d", "EFFR_ret_1d", "EWQ_France_zscore_60d", "US6M_Rate_ret_20d", "CPB_CampbellSoup_ret_5d", "heston_var_ev_h5", "DOW_Price_zscore_60d", "spx_abs_ret_max_5d", "Brent_Oil_FRED_ret_20d", "EWA_Australia_zscore_60d", "EWS_Singapore_ret_5d", "MS_MorganStanley_zscore_60d"], "is_new": true}, {"model_id": "new_h2_GLOBAL_LightGBM_N25_t1", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 2, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "3M_vol_20d", "VRP_ma5", "US1Y_Rate_ret_20d", "EQIX_Equinix_ret_5d", "CCI_CrownCastle_vol_20d", "ENB_EnbridgeInc_ret_1d", "SBUX_vol_20d", "AXP_Amex_ret_20d", "EQR_Equity_ret_1d", "EWS_Singapore_ret_5d", "DAX_Germany_vol_20d", "T_ret_1d", "VVIX_ret_20d", "Nikkei_Japan_vol_20d", "SBUX_ret_5d", "ES_Evergy_ret_1d", "ASX_Australia_ret_5d", "CTAS_Cintas_vol_20d", "heston_var_ev_h3", "LMT_LockheedMartin_ret_1d", "AMGN_Amgen_ret_1d", "heston_ev_h3", "CPB_CampbellSoup_ret_5d"], "is_new": true}, {"model_id": "new_h2_GLOBAL_LightGBM_N25_t2", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 2, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "M_Macys_vol_20d", "US5Y_Rate_ret_5d", "AMD_ret_5d", "3M_vol_20d", "spx_momentum_3d", "CPB_CampbellSoup_vol_20d", "US6M_Rate_ret_20d", "heston_var_ev_h3", "CCI_CrownCastle_vol_20d", "PG_ret_20d", "MSTR_Bitcoin3_ret_1d", "JNJ_ret_1d", "EWJ_Japan_vol_20d", "EOG_EOGResources_vol_20d", "HUM_Humana_ret_5d", "IBEX_Spain_ret_20d", "TED_Spread_zscore_60d", "EWQ_France_zscore_60d", "EXC_Exelon_ret_1d", "EWL_Switzerland_vol_20d", "SLB_Schlumberger_ret_5d", "SBUX_ret_5d", "Core_CPI_zscore_60d"], "is_new": true}, {"model_id": "new_h2_GLOBAL_LightGBM_N25_t3", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 2, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "SCHW_Schwab_ret_5d", "HangSeng_HK_ret_5d", "ORCL_vol_20d", "XLY_Disc_vol_20d", "Core_CPI_zscore_60d", "spx_abs_ret_max_5d", "PLD_Prologis_ret_5d", "XOM_ret_20d", "BDX_Becton_Dickinson_ret_20d", "US1Y_Rate_ret_20d", "PAYX_Paychex_zscore_60d", "AMD_ret_1d", "DE_Deere_vol_20d", "SBUX_ret_5d", "DHR_vol_20d", "TGT_Target_zscore_60d", "ES_Evergy_ret_1d", "ASX_Australia_ret_5d", "EXC_Exelon_ret_1d", "BA_ret_1d", "AXP_Amex_ret_20d", "EWQ_France_ret_20d", "T10Y2Y_Spread_ret_5d"], "is_new": true}, {"model_id": "new_h2_GLOBAL_LightGBM_N25_t4", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 2, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EQIX_Equinix_ret_5d", "EWG_Germany_vol_20d", "ENB_EnbridgeInc_ret_1d", "CTAS_Cintas_vol_20d", "MSTR_Bitcoin3_ret_20d", "AMD_ret_5d", "AMZN_ret_5d", "MRK_Merck_zscore_60d", "WTI_Oil_FRED_zscore_60d", "EWJ_Japan_vol_20d", "vix_mean_abs_ret_5d", "FedFunds_zscore_60d", "Brent_Oil_FRED_ret_5d", "TXN_vol_20d", "EWM_Malaysia_zscore_60d", "DHR_ret_1d", "LMT_LockheedMartin_ret_1d", "ES_Evergy_ret_1d", "PAYX_Paychex_zscore_60d", "MSTR_Bitcoin3_ret_5d", "EMR_Emerson_ret_20d", "LOW_Lowes_ret_5d", "DIS_vol_20d"], "is_new": true}, {"model_id": "new_h2_GLOBAL_LightGBM_N25_t5", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 2, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWL_Switzerland_zscore_60d", "CTAS_Cintas_vol_20d", "CPB_CampbellSoup_ret_5d", "ORCL_vol_20d", "PLD_Prologis_ret_5d", "US3M_Rate_vol_20d", "LLY_zscore_60d", "HD_zscore_60d", "ENB_EnbridgeInc_ret_1d", "TED_Spread_zscore_60d", "EWL_Switzerland_vol_20d", "heston_var_ev_h3", "MSTR_Bitcoin3_ret_5d", "SCHW_Schwab_ret_5d", "AMZN_ret_5d", "spx_vol_5d", "CI_Cigna_vol_20d", "M_Macys_vol_20d", "PAYX_Paychex_vol_20d", "SLB_Schlumberger_ret_1d", "vix_acceleration_1d", "VRP_ma5", "Nikkei_Japan_zscore_60d"], "is_new": true}, {"model_id": "new_h2_GLOBAL_LightGBM_N25_t6", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 2, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EQIX_Equinix_ret_5d", "INTC_ret_5d", "CPB_CampbellSoup_ret_20d", "IYR_US_REIT2_zscore_60d", "EWA_Australia_ret_1d", "SLB_Schlumberger_ret_1d", "MS_MorganStanley_zscore_60d", "EXC_Exelon_ret_1d", "EOG_EOGResources_ret_5d", "Nikkei_Japan_zscore_60d", "PAYX_Paychex_ret_20d", "EWM_Malaysia_zscore_60d", "AMGN_Amgen_ret_1d", "heston_var_ev_h3", "XLY_Disc_vol_20d", "AVB_AvalonBay_zscore_60d", "US3Y_Rate_ret_5d", "HangSeng_HK_ret_1d", "CI_Cigna_vol_20d", "SBUX_ret_5d", "Industrial_Production_zscore_60d", "EWH_HongKong_ret_5d", "AMT_AmericanTower_ret_1d"], "is_new": true}, {"model_id": "new_h2_GLOBAL_LightGBM_N25_t7", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 2, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "Brent_Oil_FRED_ret_5d", "XLF_Fin_vol_20d", "T10Y2Y_Spread_ret_5d", "HD_ret_1d", "vix_mean_abs_ret_5d", "BLK_BlackRock_zscore_60d", "JNJ_ret_1d", "AORD_AUS_zscore_60d", "PAYX_Paychex_vol_20d", "ITT_ITTInc_ret_5d", "INTC_ret_5d", "3M_ret_5d", "TED_Spread_zscore_60d", "EXC_Exelon_zscore_60d", "SJM_JM_Smucker_ret_5d", "gjr_condvar_h1", "EMR_Emerson_ret_20d", "T_ret_1d", "HD_zscore_60d", "NFCI_ret_5d", "SBUX_zscore_60d", "heston_var_ev_h3", "hmm_p_stress"], "is_new": true}, {"model_id": "new_h2_GLOBAL_LightGBM_N30_t0", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 2, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "PAYX_Paychex_zscore_60d", "XLK_Tech_zscore_60d", "SBUX_zscore_60d", "AXP_Amex_vol_20d", "3M_vol_20d", "US30Y_Rate_ret_20d", "EWM_Malaysia_vol_20d", "AMZN_ret_5d", "vix_mean_abs_ret_5d", "MS_MorganStanley_ret_5d", "GE_ret_1d", "HUM_Humana_ret_5d", "AVB_AvalonBay_zscore_60d", "Industrial_Production_zscore_60d", "SLB_Schlumberger_ret_5d", "EWM_Malaysia_zscore_60d", "ITT_ITTInc_ret_5d", "IWM_SmallCap_vol_20d", "spx_abs_ret_max_5d", "CI_Cigna_vol_20d", "SJM_JM_Smucker_ret_1d", "EMR_Emerson_ret_20d", "PAYX_Paychex_ret_20d", "ENB_EnbridgeInc_ret_1d", "PCAR_PaccarInc_ret_5d", "US7Y_Rate_ret_20d", "EWS_Singapore_ret_5d", "DHR_ret_1d"], "is_new": true}, {"model_id": "new_h2_GLOBAL_LightGBM_N30_t1", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 2, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "T_ret_1d", "EWQ_France_ret_20d", "DE_Deere_ret_5d", "AMD_ret_1d", "LOW_Lowes_ret_5d", "CPB_CampbellSoup_zscore_60d", "CMCSA_ret_1d", "EFFR_vol_20d", "BTI_BritishAmerican_ret_20d", "EWA_Australia_zscore_60d", "EXC_Exelon_ret_1d", "AORD_AUS_zscore_60d", "Nikkei_Japan_vol_20d", "EWC_Canada_zscore_60d", "MSTR_Bitcoin3_ret_5d", "TGT_Target_zscore_60d", "XOM_ret_20d", "Brent_Oil_FRED_ret_20d", "VVIX_ret_20d", "AVB_AvalonBay_zscore_60d", "LUV_SouthwestAir_ret_5d", "XLF_Fin_vol_20d", "EQIX_Equinix_ret_5d", "HangSeng_HK_ret_5d", "WTI_Oil_FRED_zscore_60d", "M_Macys_vol_20d", "INTC_ret_1d", "heston_var_ev_h3"], "is_new": true}, {"model_id": "new_h2_GLOBAL_LightGBM_N30_t2", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 2, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "Brent_Oil_FRED_ret_5d", "CLX_Clorox_vol_20d", "PFE_ret_1d", "AXP_Amex_vol_20d", "MS_MorganStanley_ret_5d", "GE_ret_1d", "TM_Telephone_vol_20d", "LMT_LockheedMartin_vol_20d", "ORCL_zscore_60d", "US3M_Rate_zscore_60d", "ORCL_vol_20d", "SLB_Schlumberger_ret_5d", "US1Y_Rate_ret_5d", "Michigan_Sentiment_ret_20d", "EWA_Australia_zscore_60d", "vix_acceleration_1d", "EMR_Emerson_ret_20d", "EXC_Exelon_ret_1d", "BDX_Becton_Dickinson_ret_20d", "PAYX_Paychex_vol_20d", "AMZN_ret_5d", "HangSeng_HK_vol_20d", "HangSeng_HK_ret_1d", "TXN_vol_20d", "CCI_CrownCastle_vol_20d", "EWY_Korea_ret_20d", "heston_var_ev_h7", "T10Y2Y_Spread_ret_5d"], "is_new": true}, {"model_id": "new_h2_GLOBAL_LightGBM_N30_t3", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 2, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "GILD_Gilead_ret_20d", "IYR_US_REIT2_zscore_60d", "NEE_NextEra_ret_20d", "SJM_JM_Smucker_ret_1d", "M_Macys_vol_20d", "PG_ret_20d", "ORCL_zscore_60d", "NOC_Northrop_ret_20d", "Brent_Oil_FRED_ret_20d", "US7Y_Rate_ret_20d", "Industrial_Production_zscore_60d", "DE_Deere_ret_5d", "GD_GeneralDynamics_zscore_60d", "US3M_Rate_vol_20d", "spx_abs_ret_max_5d", "AORD_AUS_zscore_60d", "CPB_CampbellSoup_ret_20d", "SBUX_ret_5d", "ASX_Australia_vol_20d", "US3M_Rate_zscore_60d", "EFFR_ret_1d", "XLF_Fin_vol_20d", "PAYX_Paychex_zscore_60d", "NWL_Newell_ret_20d", "HangSeng_HK_vol_20d", "IYM_BasicMaterials_ret_20d", "Brent_Oil_FRED_ret_5d", "XLB_Materials_zscore_60d"], "is_new": true}, {"model_id": "new_h2_GLOBAL_LightGBM_N30_t4", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 2, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "vix_acceleration_1d", "EWG_Germany_vol_20d", "US1Y_Rate_ret_5d", "3M_vol_20d", "EWY_Korea_zscore_60d", "LMT_LockheedMartin_ret_1d", "EWM_Malaysia_vol_20d", "HangSeng_HK_vol_20d", "PAYX_Paychex_zscore_60d", "SLB_Schlumberger_ret_1d", "EMR_Emerson_ret_20d", "ORCL_zscore_60d", "PFE_ret_1d", "DE_Deere_vol_20d", "ITT_ITTInc_ret_5d", "EWA_Australia_ret_1d", "DOW_Price_zscore_60d", "BLK_BlackRock_zscore_60d", "LMT_LockheedMartin_vol_20d", "VOD_Vodafone_zscore_60d", "EQR_Equity_ret_1d", "TGT_Target_zscore_60d", "GD_GeneralDynamics_zscore_60d", "TED_Spread_vol_20d", "US3M_Rate_vol_20d", "spx_vol_5d", "QQQ_vol_20d", "EWQ_France_ret_20d"], "is_new": true}, {"model_id": "new_h2_GLOBAL_LightGBM_N30_t5", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 2, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EOG_EOGResources_ret_5d", "XLY_Disc_vol_20d", "XLF_Fin_vol_20d", "EWA_Australia_ret_1d", "EWL_Switzerland_vol_20d", "XOM_ret_20d", "Industrial_Production_zscore_60d", "PPL_PPL_ret_1d", "hmm_p_stress", "AMGN_Amgen_ret_1d", "EMR_Emerson_ret_20d", "EXC_Exelon_ret_1d", "AORD_AUS_zscore_60d", "XLK_Tech_zscore_60d", "EWS_Singapore_ret_5d", "US5Y_Rate_ret_5d", "SJM_JM_Smucker_ret_5d", "M_Macys_vol_20d", "NWL_Newell_ret_20d", "EWC_Canada_zscore_60d", "TED_Spread_zscore_60d", "HangSeng_HK_vol_20d", "MSTR_Bitcoin3_ret_5d", "HD_ret_5d", "TGT_Target_zscore_60d", "US6M_Rate_ret_20d", "GD_GeneralDynamics_zscore_60d", "DE_Deere_vol_20d"], "is_new": true}, {"model_id": "new_h2_GLOBAL_LightGBM_N30_t6", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 2, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "INTC_ret_5d", "MS_MorganStanley_ret_5d", "LLY_zscore_60d", "DE_Deere_ret_5d", "EWQ_France_zscore_60d", "XLB_Materials_zscore_60d", "Brent_Oil_FRED_ret_5d", "US1Y_Rate_ret_20d", "heston_var_ev_h5", "NOC_Northrop_ret_20d", "PCAR_PaccarInc_ret_5d", "Michigan_Sentiment_ret_20d", "ENB_EnbridgeInc_ret_1d", "EFFR_ret_1d", "QQQ_vol_20d", "EWY_Korea_ret_20d", "IYR_US_REIT2_zscore_60d", "EWM_Malaysia_ret_1d", "EWQ_France_ret_20d", "IBEX_Spain_ret_20d", "3M_ret_5d", "MS_MorganStanley_ret_1d", "EFFR_vol_20d", "BDX_Becton_Dickinson_ret_20d", "CPB_CampbellSoup_vol_20d", "NFCI_ret_5d", "NWL_Newell_ret_20d", "EWA_Australia_zscore_60d"], "is_new": true}, {"model_id": "new_h2_GLOBAL_LightGBM_N30_t7", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 2, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "FedFunds_zscore_60d", "DE_Deere_vol_20d", "CMCSA_ret_1d", "MRK_Merck_zscore_60d", "MS_MorganStanley_zscore_60d", "EWC_Canada_zscore_60d", "EXC_Exelon_ret_1d", "JNJ_ret_1d", "EQIX_Equinix_ret_5d", "NOC_Northrop_ret_20d", "EWJ_Japan_vol_20d", "M_Macys_vol_20d", "EOG_EOGResources_ret_5d", "TED_Spread_zscore_60d", "US1Y_Rate_ret_5d", "NVDA_vol_20d", "EQR_Equity_ret_1d", "EWL_Switzerland_vol_20d", "EWA_Australia_zscore_60d", "LUV_SouthwestAir_ret_5d", "PFE_ret_1d", "ASX_Australia_ret_5d", "HD_ret_20d", "GILD_Gilead_ret_20d", "GD_GeneralDynamics_zscore_60d", "VVIX_ret_20d", "LOW_Lowes_ret_5d", "Core_PCE_zscore_60d"], "is_new": true}, {"model_id": "new_h2_GLOBAL_GradientBoosting_N5_t0", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 2, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "HD_zscore_60d", "DIS_vol_20d", "CI_Cigna_vol_20d"], "is_new": true}, {"model_id": "new_h2_GLOBAL_GradientBoosting_N5_t1", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 2, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "FedFunds_zscore_60d", "US5Y_Rate_ret_5d", "HUM_Humana_ret_5d"], "is_new": true}, {"model_id": "new_h2_GLOBAL_GradientBoosting_N5_t2", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 2, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "ENB_EnbridgeInc_ret_1d", "CCI_CrownCastle_vol_20d", "gjr_condvar_h1"], "is_new": true}, {"model_id": "new_h2_GLOBAL_GradientBoosting_N5_t3", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 2, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "MSTR_Bitcoin3_ret_1d", "EWQ_France_ret_20d", "SLB_Schlumberger_ret_1d"], "is_new": true}, {"model_id": "new_h2_GLOBAL_GradientBoosting_N5_t4", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 2, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "XOM_ret_1d", "CTAS_Cintas_vol_20d", "DAX_Germany_vol_20d"], "is_new": true}, {"model_id": "new_h2_GLOBAL_GradientBoosting_N5_t5", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 2, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWY_Korea_zscore_60d", "spx_vol_5d", "MS_MorganStanley_ret_1d"], "is_new": true}, {"model_id": "new_h2_GLOBAL_GradientBoosting_N5_t6", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 2, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWY_Korea_zscore_60d", "ORCL_zscore_60d", "EWG_Germany_vol_20d"], "is_new": true}, {"model_id": "new_h2_GLOBAL_GradientBoosting_N5_t7", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 2, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "heston_var_ev_h7", "US3M_Rate_zscore_60d", "Core_PCE_zscore_60d"], "is_new": true}, {"model_id": "new_h2_GLOBAL_GradientBoosting_N8_t0", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 2, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "PG_ret_20d", "INTC_ret_1d", "Core_PCE_zscore_60d", "EXC_Exelon_zscore_60d", "MO_AltriaMG_ret_1d", "hmm_p_stress"], "is_new": true}, {"model_id": "new_h2_GLOBAL_GradientBoosting_N8_t1", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 2, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "US3M_Rate_zscore_60d", "EQIX_Equinix_ret_5d", "EWQ_France_ret_20d", "AXP_Amex_vol_20d", "Industrial_Production_zscore_60d", "PLD_Prologis_ret_5d"], "is_new": true}, {"model_id": "new_h2_GLOBAL_GradientBoosting_N8_t2", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 2, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWC_Canada_zscore_60d", "Brent_Oil_FRED_ret_5d", "AMZN_ret_5d", "SJM_JM_Smucker_ret_5d", "CMCSA_ret_1d", "DHR_ret_1d"], "is_new": true}, {"model_id": "new_h2_GLOBAL_GradientBoosting_N8_t3", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 2, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "Retail_Sales_zscore_60d", "TGT_Target_zscore_60d", "heston_var_ev_h3", "MSTR_Bitcoin3_ret_5d", "EOG_EOGResources_ret_5d", "HangSeng_HK_vol_20d"], "is_new": true}, {"model_id": "new_h2_GLOBAL_GradientBoosting_N8_t4", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 2, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "gjr_condvar_h1", "NOC_Northrop_ret_20d", "AMD_ret_5d", "NWL_Newell_ret_20d", "NVDA_vol_20d", "VRP_ma5"], "is_new": true}, {"model_id": "new_h2_GLOBAL_GradientBoosting_N8_t5", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 2, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "VVIX_ret_20d", "XOM_ret_20d", "NFCI_ret_5d", "EWS_Singapore_ret_5d", "PAYX_Paychex_vol_20d", "IBEX_Spain_ret_20d"], "is_new": true}, {"model_id": "new_h2_GLOBAL_GradientBoosting_N8_t6", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 2, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "SPY_zscore_60d", "SBUX_zscore_60d", "WTI_Oil_FRED_zscore_60d", "XOM_ret_20d", "ASX_Australia_vol_20d", "MSTR_Bitcoin3_ret_5d"], "is_new": true}, {"model_id": "new_h2_GLOBAL_GradientBoosting_N8_t7", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 2, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "GD_GeneralDynamics_zscore_60d", "ORCL_zscore_60d", "CI_Cigna_vol_20d", "LMT_LockheedMartin_ret_1d", "US6M_Rate_ret_20d", "AMD_ret_1d"], "is_new": true}, {"model_id": "new_h2_GLOBAL_GradientBoosting_N10_t0", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 2, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "3M_ret_5d", "DHR_ret_1d", "EWA_Australia_ret_1d", "CCI_CrownCastle_vol_20d", "NOC_Northrop_ret_20d", "AXP_Amex_ret_20d", "EXC_Exelon_ret_1d", "BA_ret_1d"], "is_new": true}, {"model_id": "new_h2_GLOBAL_GradientBoosting_N10_t1", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 2, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "SLB_Schlumberger_ret_5d", "EWM_Malaysia_vol_20d", "heston_var_ev_h3", "DE_Deere_vol_20d", "CTAS_Cintas_vol_20d", "EOG_EOGResources_vol_20d", "BLK_BlackRock_zscore_60d", "US1Y_Rate_ret_20d"], "is_new": true}, {"model_id": "new_h2_GLOBAL_GradientBoosting_N10_t2", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 2, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "DHR_vol_20d", "MRK_Merck_zscore_60d", "ASX_Australia_vol_20d", "PG_ret_20d", "XLY_Disc_vol_20d", "MS_MorganStanley_zscore_60d", "MSTR_Bitcoin3_ret_20d", "XLB_Materials_zscore_60d"], "is_new": true}, {"model_id": "new_h2_GLOBAL_GradientBoosting_N10_t3", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 2, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "ASX_Australia_ret_5d", "BLK_BlackRock_zscore_60d", "heston_var_ev_h7", "BTI_BritishAmerican_ret_5d", "HD_ret_5d", "CI_Cigna_vol_20d", "HD_ret_1d", "GE_ret_1d"], "is_new": true}, {"model_id": "new_h2_GLOBAL_GradientBoosting_N10_t4", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 2, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "heston_var_ev_h5", "QQQ_vol_20d", "EWM_Malaysia_ret_1d", "SBUX_zscore_60d", "EOG_EOGResources_ret_5d", "EWY_Korea_ret_20d", "TGT_Target_zscore_60d", "NWL_Newell_ret_20d"], "is_new": true}, {"model_id": "new_h2_GLOBAL_GradientBoosting_N10_t5", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 2, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "SPY_zscore_60d", "ITT_ITTInc_ret_5d", "WTI_Oil_FRED_zscore_60d", "GE_ret_1d", "PCAR_PaccarInc_ret_5d", "spx_momentum_3d", "PPL_PPL_ret_1d", "ORCL_vol_20d"], "is_new": true}, {"model_id": "new_h2_GLOBAL_GradientBoosting_N10_t6", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 2, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "AORD_AUS_zscore_60d", "JNJ_ret_1d", "INTC_ret_1d", "SJM_JM_Smucker_ret_1d", "Retail_Sales_zscore_60d", "US5Y_Rate_ret_5d", "PPL_PPL_ret_1d", "spx_momentum_3d"], "is_new": true}, {"model_id": "new_h2_GLOBAL_GradientBoosting_N10_t7", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 2, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "ORCL_vol_20d", "AMGN_Amgen_ret_1d", "PPL_PPL_ret_1d", "IBEX_Spain_ret_20d", "XOM_ret_20d", "CPB_CampbellSoup_ret_5d", "AVB_AvalonBay_zscore_60d", "MO_AltriaMG_ret_1d"], "is_new": true}, {"model_id": "new_h2_GLOBAL_GradientBoosting_N12_t0", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 2, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "INTC_ret_5d", "3M_vol_20d", "HD_ret_1d", "EWM_Malaysia_vol_20d", "US3M_Rate_zscore_60d", "BDX_Becton_Dickinson_ret_20d", "EWQ_France_ret_20d", "JNJ_ret_1d", "ORCL_vol_20d", "AMZN_ret_5d"], "is_new": true}, {"model_id": "new_h2_GLOBAL_GradientBoosting_N12_t1", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 2, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "NWL_Newell_ret_20d", "AMD_ret_5d", "XLY_Disc_vol_20d", "NVDA_vol_20d", "EWJ_Japan_vol_20d", "LOW_Lowes_ret_20d", "HangSeng_HK_vol_20d", "PLD_Prologis_ret_5d", "MS_MorganStanley_ret_5d", "AMGN_Amgen_ret_1d"], "is_new": true}, {"model_id": "new_h2_GLOBAL_GradientBoosting_N12_t2", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 2, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "CLX_Clorox_vol_20d", "BDX_Becton_Dickinson_ret_20d", "SJM_JM_Smucker_ret_5d", "HD_zscore_60d", "DE_Deere_ret_5d", "MS_MorganStanley_ret_1d", "ENB_EnbridgeInc_ret_1d", "spx_vol_5d", "DHR_ret_1d", "MSTR_Bitcoin3_ret_20d"], "is_new": true}, {"model_id": "new_h2_GLOBAL_GradientBoosting_N12_t3", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 2, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "DE_Deere_ret_5d", "HD_ret_1d", "LLY_zscore_60d", "US7Y_Rate_ret_20d", "HUM_Humana_ret_5d", "BLK_BlackRock_zscore_60d", "SJM_JM_Smucker_ret_5d", "LMT_LockheedMartin_ret_1d", "XLF_Fin_vol_20d", "heston_ev_h3"], "is_new": true}, {"model_id": "new_h2_GLOBAL_GradientBoosting_N12_t4", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 2, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWQ_France_zscore_60d", "Industrial_Production_zscore_60d", "EFFR_vol_20d", "GILD_Gilead_ret_20d", "heston_var_ev_h5", "EWS_Singapore_ret_5d", "PPL_PPL_ret_1d", "PCAR_PaccarInc_ret_5d", "VRP_ma5", "IYM_BasicMaterials_ret_20d"], "is_new": true}, {"model_id": "new_h2_GLOBAL_GradientBoosting_N12_t5", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 2, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "ITT_ITTInc_ret_5d", "PFE_ret_1d", "EWM_Malaysia_ret_1d", "SBUX_vol_20d", "EWY_Korea_zscore_60d", "US30Y_Rate_ret_20d", "LMT_LockheedMartin_vol_20d", "EFFR_ret_1d", "IYM_BasicMaterials_ret_20d", "Brent_Oil_FRED_ret_5d"], "is_new": true}, {"model_id": "new_h2_GLOBAL_GradientBoosting_N12_t6", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 2, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "NVDA_vol_20d", "PFE_ret_1d", "VRP_ma5", "heston_var_ev_h3", "BA_ret_1d", "CI_Cigna_vol_20d", "spx_momentum_3d", "SBUX_zscore_60d", "IWM_SmallCap_vol_20d", "EOG_EOGResources_ret_5d"], "is_new": true}, {"model_id": "new_h2_GLOBAL_GradientBoosting_N12_t7", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 2, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "US3M_Rate_vol_20d", "Nikkei_Japan_vol_20d", "IWM_SmallCap_vol_20d", "EMR_Emerson_ret_20d", "EWH_HongKong_ret_5d", "XOM_ret_1d", "MS_MorganStanley_zscore_60d", "SJM_JM_Smucker_ret_1d", "EXC_Exelon_ret_1d", "US7Y_Rate_ret_20d"], "is_new": true}, {"model_id": "new_h2_GLOBAL_GradientBoosting_N15_t0", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 2, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWM_Malaysia_ret_1d", "vix_acceleration_1d", "SPY_zscore_60d", "Industrial_Production_zscore_60d", "GD_GeneralDynamics_zscore_60d", "ASX_Australia_vol_20d", "ORCL_zscore_60d", "3M_ret_5d", "BLK_BlackRock_zscore_60d", "US30Y_Rate_ret_20d", "FedFunds_zscore_60d", "CLX_Clorox_vol_20d", "JNJ_ret_1d"], "is_new": true}, {"model_id": "new_h2_GLOBAL_GradientBoosting_N15_t1", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 2, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "DHR_vol_20d", "NOC_Northrop_ret_20d", "US3M_Rate_vol_20d", "INTC_ret_5d", "PPL_PPL_ret_1d", "EWL_Switzerland_zscore_60d", "TXN_vol_20d", "JNJ_ret_1d", "BTI_BritishAmerican_ret_5d", "EWG_Germany_ret_20d", "XLF_Fin_vol_20d", "SBUX_ret_5d", "MSTR_Bitcoin3_ret_1d"], "is_new": true}, {"model_id": "new_h2_GLOBAL_GradientBoosting_N15_t2", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 2, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "XLV_Health_zscore_60d", "HD_ret_20d", "NOC_Northrop_ret_20d", "Brent_Oil_FRED_ret_20d", "IBEX_Spain_ret_20d", "DE_Deere_ret_5d", "ORCL_vol_20d", "ENB_EnbridgeInc_ret_1d", "AMT_AmericanTower_ret_1d", "GE_ret_1d", "BLK_BlackRock_zscore_60d", "SLB_Schlumberger_ret_5d", "LMT_LockheedMartin_vol_20d"], "is_new": true}, {"model_id": "new_h2_GLOBAL_GradientBoosting_N15_t3", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 2, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWG_Germany_vol_20d", "FedFunds_zscore_60d", "NEE_NextEra_ret_20d", "Brent_Oil_FRED_ret_5d", "BTI_BritishAmerican_ret_5d", "TED_Spread_vol_20d", "WTI_Oil_FRED_zscore_60d", "SJM_JM_Smucker_ret_1d", "SCHW_Schwab_ret_5d", "EWA_Australia_ret_1d", "PAYX_Paychex_vol_20d", "DHR_vol_20d", "T10Y2Y_Spread_ret_5d"], "is_new": true}, {"model_id": "new_h2_GLOBAL_GradientBoosting_N15_t4", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 2, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "spx_momentum_3d", "EWL_Switzerland_vol_20d", "SLB_Schlumberger_ret_1d", "AVB_AvalonBay_zscore_60d", "HD_ret_5d", "CPB_CampbellSoup_vol_20d", "XLK_Tech_zscore_60d", "BTI_BritishAmerican_ret_20d", "SBUX_ret_5d", "CI_Cigna_vol_20d", "MSTR_Bitcoin3_ret_5d", "AXP_Amex_vol_20d", "EWS_Singapore_ret_5d"], "is_new": true}, {"model_id": "new_h2_GLOBAL_GradientBoosting_N15_t5", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 2, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "HD_zscore_60d", "HD_ret_20d", "CLX_Clorox_vol_20d", "TED_Spread_zscore_60d", "hmm_p_stress", "HangSeng_HK_ret_5d", "EWY_Korea_ret_20d", "NVDA_vol_20d", "spx_abs_ret_max_5d", "Nikkei_Japan_zscore_60d", "BA_ret_1d", "vix_acceleration_1d", "EQR_Equity_ret_1d"], "is_new": true}, {"model_id": "new_h2_GLOBAL_GradientBoosting_N15_t6", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 2, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWL_Switzerland_zscore_60d", "PPL_PPL_ret_1d", "US1Y_Rate_ret_20d", "TM_Telephone_vol_20d", "AXP_Amex_vol_20d", "EFFR_vol_20d", "BTI_BritishAmerican_ret_5d", "ASX_Australia_ret_5d", "US6M_Rate_ret_20d", "EWQ_France_zscore_60d", "NOC_Northrop_ret_20d", "SBUX_vol_20d", "NFCI_ret_5d"], "is_new": true}, {"model_id": "new_h2_GLOBAL_GradientBoosting_N15_t7", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 2, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWQ_France_zscore_60d", "NOC_Northrop_ret_20d", "JNJ_ret_1d", "PCAR_PaccarInc_ret_5d", "MSTR_Bitcoin3_ret_1d", "IBEX_Spain_ret_20d", "LOW_Lowes_ret_20d", "LMT_LockheedMartin_ret_1d", "HangSeng_HK_ret_1d", "IYM_BasicMaterials_ret_20d", "XLY_Disc_vol_20d", "BDX_Becton_Dickinson_ret_20d", "EWH_HongKong_ret_5d"], "is_new": true}, {"model_id": "new_h2_GLOBAL_GradientBoosting_N20_t0", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 2, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWG_Germany_ret_20d", "LOW_Lowes_ret_20d", "CTAS_Cintas_vol_20d", "CPB_CampbellSoup_zscore_60d", "AXP_Amex_vol_20d", "DHR_ret_1d", "MSTR_Bitcoin3_ret_20d", "XOM_ret_1d", "EWG_Germany_vol_20d", "US3Y_Rate_ret_5d", "INTC_ret_1d", "MRK_Merck_zscore_60d", "EWQ_France_ret_20d", "TM_Telephone_ret_1d", "XLY_Disc_vol_20d", "DE_Deere_vol_20d", "EWA_Australia_zscore_60d", "EWY_Korea_ret_20d"], "is_new": true}, {"model_id": "new_h2_GLOBAL_GradientBoosting_N20_t1", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 2, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "BTI_BritishAmerican_ret_5d", "IBEX_Spain_ret_20d", "spx_abs_ret_max_5d", "T10Y2Y_Spread_ret_5d", "INTC_ret_1d", "Industrial_Production_zscore_60d", "AORD_AUS_zscore_60d", "PG_ret_20d", "AMT_AmericanTower_ret_1d", "DHR_vol_20d", "MSTR_Bitcoin3_ret_1d", "EWY_Korea_zscore_60d", "AMGN_Amgen_ret_1d", "Nikkei_Japan_zscore_60d", "DHR_ret_1d", "NWL_Newell_ret_20d", "AMD_ret_5d", "MSTR_Bitcoin3_ret_5d"], "is_new": true}, {"model_id": "new_h2_GLOBAL_GradientBoosting_N20_t2", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 2, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "gjr_condvar_h1", "DAX_Germany_vol_20d", "HangSeng_HK_ret_1d", "HangSeng_HK_vol_20d", "SLB_Schlumberger_ret_1d", "Retail_Sales_zscore_60d", "ORCL_vol_20d", "EFFR_vol_20d", "CPB_CampbellSoup_zscore_60d", "NVDA_vol_20d", "HD_ret_5d", "US30Y_Rate_ret_20d", "EWS_Singapore_ret_5d", "EXC_Exelon_ret_1d", "spx_vol_5d", "DE_Deere_vol_20d", "PCAR_PaccarInc_ret_5d", "US6M_Rate_ret_20d"], "is_new": true}, {"model_id": "new_h2_GLOBAL_GradientBoosting_N20_t3", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 2, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "US7Y_Rate_ret_20d", "HD_ret_20d", "BA_ret_1d", "ORCL_vol_20d", "US3M_Rate_zscore_60d", "IWM_SmallCap_vol_20d", "Retail_Sales_zscore_60d", "EOG_EOGResources_ret_5d", "LLY_zscore_60d", "ENB_EnbridgeInc_ret_1d", "vix_acceleration_1d", "ASX_Australia_ret_5d", "EFFR_vol_20d", "EWM_Malaysia_vol_20d", "LOW_Lowes_ret_5d", "SLB_Schlumberger_ret_1d", "XLY_Disc_vol_20d", "EWG_Germany_vol_20d"], "is_new": true}, {"model_id": "new_h2_GLOBAL_GradientBoosting_N20_t4", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 2, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "MRK_Merck_zscore_60d", "TXN_vol_20d", "EQR_Equity_ret_1d", "TM_Telephone_ret_1d", "FedFunds_zscore_60d", "TED_Spread_zscore_60d", "US30Y_Rate_ret_20d", "XLB_Materials_zscore_60d", "CI_Cigna_vol_20d", "VRP_ma5", "vix_mean_abs_ret_5d", "VOD_Vodafone_zscore_60d", "HangSeng_HK_ret_1d", "LOW_Lowes_ret_20d", "MSTR_Bitcoin3_ret_5d", "IYR_US_REIT2_zscore_60d", "XOM_ret_1d", "US7Y_Rate_ret_20d"], "is_new": true}, {"model_id": "new_h2_GLOBAL_GradientBoosting_N20_t5", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 2, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "spx_momentum_3d", "HUM_Humana_ret_5d", "HangSeng_HK_ret_1d", "Nikkei_Japan_zscore_60d", "EWY_Korea_zscore_60d", "ITT_ITTInc_ret_5d", "DHR_vol_20d", "EWQ_France_ret_20d", "US7Y_Rate_ret_20d", "3M_vol_20d", "AORD_AUS_zscore_60d", "EXC_Exelon_zscore_60d", "Brent_Oil_FRED_ret_20d", "MO_AltriaMG_ret_1d", "EOG_EOGResources_ret_5d", "HangSeng_HK_ret_5d", "EFFR_ret_1d", "TED_Spread_vol_20d"], "is_new": true}, {"model_id": "new_h2_GLOBAL_GradientBoosting_N20_t6", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 2, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EXC_Exelon_ret_1d", "SCHW_Schwab_ret_5d", "AMD_ret_1d", "BTI_BritishAmerican_ret_5d", "SJM_JM_Smucker_ret_5d", "VOD_Vodafone_zscore_60d", "PPL_PPL_ret_1d", "SBUX_vol_20d", "WTI_Oil_FRED_zscore_60d", "LLY_zscore_60d", "AXP_Amex_ret_20d", "DAX_Germany_vol_20d", "HangSeng_HK_ret_5d", "MSTR_Bitcoin3_ret_1d", "PAYX_Paychex_zscore_60d", "XLV_Health_zscore_60d", "EWM_Malaysia_vol_20d", "NWL_Newell_ret_20d"], "is_new": true}, {"model_id": "new_h2_GLOBAL_GradientBoosting_N20_t7", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 2, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "DAX_Germany_zscore_60d", "SJM_JM_Smucker_ret_5d", "AORD_AUS_zscore_60d", "BDX_Becton_Dickinson_ret_20d", "IBEX_Spain_ret_20d", "vix_acceleration_1d", "Brent_Oil_FRED_ret_5d", "PG_ret_20d", "SBUX_ret_5d", "ASX_Australia_ret_5d", "heston_ev_h3", "SCHW_Schwab_ret_5d", "EWQ_France_zscore_60d", "GE_ret_1d", "NWL_Newell_ret_20d", "AVB_AvalonBay_zscore_60d", "EWY_Korea_ret_20d", "CPB_CampbellSoup_zscore_60d"], "is_new": true}, {"model_id": "new_h2_GLOBAL_GradientBoosting_N25_t0", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 2, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "AMD_ret_5d", "SBUX_vol_20d", "VOD_Vodafone_zscore_60d", "XLY_Disc_vol_20d", "DAX_Germany_zscore_60d", "CLX_Clorox_vol_20d", "MSTR_Bitcoin3_ret_5d", "GD_GeneralDynamics_zscore_60d", "LMT_LockheedMartin_ret_1d", "SBUX_ret_5d", "ITT_ITTInc_ret_5d", "EQIX_Equinix_ret_5d", "vix_acceleration_1d", "LLY_zscore_60d", "HD_ret_5d", "XOM_ret_20d", "SJM_JM_Smucker_ret_5d", "EWG_Germany_ret_20d", "spx_abs_ret_max_5d", "US30Y_Rate_ret_20d", "M_Macys_vol_20d", "SBUX_zscore_60d", "XLF_Fin_vol_20d"], "is_new": true}, {"model_id": "new_h2_GLOBAL_GradientBoosting_N25_t1", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 2, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "NWL_Newell_ret_20d", "MS_MorganStanley_ret_5d", "T10Y2Y_Spread_ret_5d", "SLB_Schlumberger_ret_5d", "BTI_BritishAmerican_ret_5d", "LLY_zscore_60d", "LOW_Lowes_ret_20d", "PLD_Prologis_ret_5d", "CTAS_Cintas_vol_20d", "IWM_SmallCap_vol_20d", "LUV_SouthwestAir_ret_5d", "NOC_Northrop_ret_20d", "XOM_ret_20d", "CCI_CrownCastle_vol_20d", "NVDA_vol_20d", "XOM_ret_1d", "US30Y_Rate_ret_20d", "US7Y_Rate_ret_20d", "HangSeng_HK_ret_5d", "AXP_Amex_ret_20d", "US3M_Rate_vol_20d", "EWQ_France_zscore_60d", "XLV_Health_zscore_60d"], "is_new": true}, {"model_id": "new_h2_GLOBAL_GradientBoosting_N25_t2", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 2, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "NEE_NextEra_ret_20d", "heston_var_ev_h3", "AMD_ret_5d", "JNJ_ret_1d", "US3M_Rate_zscore_60d", "MO_AltriaMG_ret_1d", "INTC_ret_1d", "Core_PCE_zscore_60d", "NFCI_ret_5d", "Core_CPI_zscore_60d", "Brent_Oil_FRED_ret_20d", "EWL_Switzerland_zscore_60d", "EWH_HongKong_ret_5d", "VOD_Vodafone_zscore_60d", "IYR_US_REIT2_zscore_60d", "NOC_Northrop_ret_20d", "EWA_Australia_ret_1d", "ES_Evergy_ret_1d", "DE_Deere_ret_5d", "EXC_Exelon_zscore_60d", "MRK_Merck_zscore_60d", "Brent_Oil_FRED_ret_5d", "SLB_Schlumberger_ret_5d"], "is_new": true}, {"model_id": "new_h2_GLOBAL_GradientBoosting_N25_t3", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 2, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "BTI_BritishAmerican_ret_5d", "VRP_ma5", "DAX_Germany_zscore_60d", "CCI_CrownCastle_vol_20d", "DIS_vol_20d", "heston_var_ev_h5", "Nikkei_Japan_vol_20d", "US3M_Rate_zscore_60d", "PAYX_Paychex_zscore_60d", "IBEX_Spain_ret_20d", "US1Y_Rate_ret_5d", "MS_MorganStanley_ret_1d", "US6M_Rate_ret_20d", "SBUX_vol_20d", "US1Y_Rate_ret_20d", "HangSeng_HK_ret_5d", "TED_Spread_vol_20d", "SBUX_ret_5d", "US3Y_Rate_ret_5d", "SLB_Schlumberger_ret_1d", "NEE_NextEra_ret_20d", "EWC_Canada_zscore_60d", "CI_Cigna_vol_20d"], "is_new": true}, {"model_id": "new_h2_GLOBAL_GradientBoosting_N25_t4", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 2, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "LLY_zscore_60d", "EMR_Emerson_ret_20d", "SCHW_Schwab_ret_5d", "XOM_ret_20d", "BA_ret_1d", "EXC_Exelon_zscore_60d", "EWG_Germany_vol_20d", "MS_MorganStanley_ret_1d", "US3Y_Rate_ret_5d", "EQR_Equity_ret_1d", "heston_ev_h3", "vix_mean_abs_ret_5d", "US3M_Rate_vol_20d", "CCI_CrownCastle_vol_20d", "NWL_Newell_ret_20d", "EWY_Korea_zscore_60d", "DE_Deere_vol_20d", "EWQ_France_ret_20d", "3M_vol_20d", "Core_PCE_zscore_60d", "XLF_Fin_vol_20d", "SBUX_ret_5d", "HD_ret_20d"], "is_new": true}, {"model_id": "new_h2_GLOBAL_GradientBoosting_N25_t5", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 2, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "MS_MorganStanley_ret_1d", "EWG_Germany_vol_20d", "CPB_CampbellSoup_zscore_60d", "XLF_Fin_vol_20d", "CMCSA_ret_1d", "US3Y_Rate_ret_5d", "SJM_JM_Smucker_ret_5d", "NWL_Newell_ret_20d", "EWS_Singapore_ret_5d", "PLD_Prologis_ret_5d", "AMGN_Amgen_ret_1d", "TED_Spread_vol_20d", "LOW_Lowes_ret_5d", "LLY_zscore_60d", "EWM_Malaysia_ret_1d", "vix_mean_abs_ret_5d", "3M_vol_20d", "NVDA_vol_20d", "DOW_Price_zscore_60d", "EOG_EOGResources_vol_20d", "EWA_Australia_zscore_60d", "AMD_ret_5d", "NFCI_ret_5d"], "is_new": true}, {"model_id": "new_h2_GLOBAL_GradientBoosting_N25_t6", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 2, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWG_Germany_vol_20d", "heston_var_ev_h5", "LMT_LockheedMartin_vol_20d", "SBUX_zscore_60d", "PCAR_PaccarInc_ret_5d", "MRK_Merck_zscore_60d", "MS_MorganStanley_ret_5d", "XLY_Disc_vol_20d", "HD_zscore_60d", "ENB_EnbridgeInc_ret_1d", "XLF_Fin_vol_20d", "T10Y2Y_Spread_ret_5d", "IBEX_Spain_ret_20d", "BTI_BritishAmerican_ret_20d", "3M_ret_5d", "HangSeng_HK_ret_1d", "AMD_ret_1d", "ASX_Australia_vol_20d", "EWL_Switzerland_vol_20d", "EFFR_vol_20d", "PPL_PPL_ret_1d", "vix_mean_abs_ret_5d", "AMGN_Amgen_ret_1d"], "is_new": true}, {"model_id": "new_h2_GLOBAL_GradientBoosting_N25_t7", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 2, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "PAYX_Paychex_ret_20d", "HangSeng_HK_ret_5d", "MSTR_Bitcoin3_ret_20d", "EWG_Germany_vol_20d", "IWM_SmallCap_vol_20d", "DAX_Germany_zscore_60d", "EWM_Malaysia_vol_20d", "JNJ_ret_1d", "NFCI_ret_5d", "CCI_CrownCastle_vol_20d", "Core_CPI_zscore_60d", "US1Y_Rate_ret_5d", "SO_SouthernCo_ret_5d", "DOW_Price_zscore_60d", "AMD_ret_5d", "EWS_Singapore_ret_5d", "TM_Telephone_vol_20d", "3M_ret_5d", "Nikkei_Japan_vol_20d", "US7Y_Rate_ret_20d", "IYM_BasicMaterials_ret_20d", "SLB_Schlumberger_ret_1d", "heston_var_ev_h3"], "is_new": true}, {"model_id": "new_h2_GLOBAL_GradientBoosting_N30_t0", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 2, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "hmm_p_stress", "vix_acceleration_1d", "BTI_BritishAmerican_ret_20d", "CPB_CampbellSoup_vol_20d", "CPB_CampbellSoup_ret_20d", "EMR_Emerson_ret_20d", "SJM_JM_Smucker_ret_5d", "CCI_CrownCastle_vol_20d", "PLD_Prologis_ret_5d", "SBUX_vol_20d", "XLB_Materials_zscore_60d", "AORD_AUS_zscore_60d", "EWH_HongKong_ret_5d", "MS_MorganStanley_ret_5d", "NWL_Newell_ret_20d", "MSTR_Bitcoin3_ret_5d", "XLY_Disc_vol_20d", "EWM_Malaysia_vol_20d", "XLF_Fin_vol_20d", "HangSeng_HK_ret_5d", "Retail_Sales_zscore_60d", "EOG_EOGResources_vol_20d", "SLB_Schlumberger_ret_1d", "Brent_Oil_FRED_ret_20d", "EWY_Korea_ret_20d", "EFFR_vol_20d", "NOC_Northrop_ret_20d", "IWM_SmallCap_vol_20d"], "is_new": true}, {"model_id": "new_h2_GLOBAL_GradientBoosting_N30_t1", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 2, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWA_Australia_zscore_60d", "EWG_Germany_ret_20d", "SBUX_zscore_60d", "XLK_Tech_zscore_60d", "AXP_Amex_vol_20d", "EOG_EOGResources_vol_20d", "WTI_Oil_FRED_zscore_60d", "HD_ret_1d", "AORD_AUS_zscore_60d", "EWM_Malaysia_ret_1d", "spx_vol_5d", "LOW_Lowes_ret_5d", "EWL_Switzerland_vol_20d", "LLY_zscore_60d", "EWC_Canada_zscore_60d", "CPB_CampbellSoup_zscore_60d", "vix_acceleration_1d", "EFFR_ret_1d", "EWY_Korea_zscore_60d", "JNJ_ret_1d", "DAX_Germany_zscore_60d", "PAYX_Paychex_vol_20d", "IYM_BasicMaterials_ret_20d", "NOC_Northrop_ret_20d", "PAYX_Paychex_zscore_60d", "CPB_CampbellSoup_ret_5d", "VVIX_ret_20d", "spx_abs_ret_max_5d"], "is_new": true}, {"model_id": "new_h2_GLOBAL_GradientBoosting_N30_t2", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 2, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "Michigan_Sentiment_ret_20d", "SJM_JM_Smucker_ret_5d", "PCAR_PaccarInc_ret_5d", "US1Y_Rate_ret_5d", "BLK_BlackRock_zscore_60d", "spx_vol_5d", "BTI_BritishAmerican_ret_5d", "DAX_Germany_zscore_60d", "MS_MorganStanley_zscore_60d", "IYM_BasicMaterials_ret_20d", "DOW_Price_zscore_60d", "HD_ret_20d", "XOM_ret_1d", "HD_zscore_60d", "TM_Telephone_vol_20d", "XLB_Materials_zscore_60d", "Brent_Oil_FRED_ret_20d", "vix_acceleration_1d", "LOW_Lowes_ret_20d", "EFFR_vol_20d", "SLB_Schlumberger_ret_1d", "EWQ_France_ret_20d", "VRP_ma5", "XLK_Tech_zscore_60d", "EOG_EOGResources_ret_5d", "3M_vol_20d", "XLF_Fin_vol_20d", "IWM_SmallCap_vol_20d"], "is_new": true}, {"model_id": "new_h2_GLOBAL_GradientBoosting_N30_t3", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 2, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "SJM_JM_Smucker_ret_1d", "NFCI_ret_5d", "GD_GeneralDynamics_zscore_60d", "US5Y_Rate_ret_5d", "CLX_Clorox_vol_20d", "EWS_Singapore_ret_5d", "M_Macys_vol_20d", "CI_Cigna_vol_20d", "NEE_NextEra_ret_20d", "LMT_LockheedMartin_vol_20d", "ORCL_zscore_60d", "vix_acceleration_1d", "LLY_zscore_60d", "DHR_ret_1d", "SLB_Schlumberger_ret_5d", "ENB_EnbridgeInc_ret_1d", "DE_Deere_vol_20d", "MS_MorganStanley_ret_1d", "CPB_CampbellSoup_zscore_60d", "US3M_Rate_zscore_60d", "IWM_SmallCap_vol_20d", "spx_vol_5d", "US3M_Rate_vol_20d", "AMZN_ret_5d", "DE_Deere_ret_5d", "US7Y_Rate_ret_20d", "HangSeng_HK_vol_20d", "US30Y_Rate_ret_20d"], "is_new": true}, {"model_id": "new_h2_GLOBAL_GradientBoosting_N30_t4", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 2, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "INTC_ret_5d", "PFE_ret_1d", "Retail_Sales_zscore_60d", "NEE_NextEra_ret_20d", "MS_MorganStanley_zscore_60d", "Brent_Oil_FRED_ret_5d", "EWQ_France_ret_20d", "TM_Telephone_vol_20d", "TM_Telephone_ret_1d", "ORCL_vol_20d", "JNJ_ret_1d", "XOM_ret_20d", "gjr_condvar_h1", "AMGN_Amgen_ret_1d", "ASX_Australia_ret_5d", "US7Y_Rate_ret_20d", "CMCSA_ret_1d", "MO_AltriaMG_ret_1d", "VVIX_ret_20d", "LLY_zscore_60d", "M_Macys_vol_20d", "PAYX_Paychex_ret_20d", "SO_SouthernCo_ret_5d", "EWA_Australia_ret_1d", "DHR_vol_20d", "EWH_HongKong_ret_5d", "US3M_Rate_vol_20d", "LUV_SouthwestAir_ret_5d"], "is_new": true}, {"model_id": "new_h2_GLOBAL_GradientBoosting_N30_t5", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 2, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "Core_CPI_zscore_60d", "US6M_Rate_ret_20d", "HD_ret_20d", "CLX_Clorox_vol_20d", "EFFR_ret_1d", "heston_var_ev_h3", "PFE_ret_1d", "AORD_AUS_zscore_60d", "NOC_Northrop_ret_20d", "MSTR_Bitcoin3_ret_5d", "CPB_CampbellSoup_zscore_60d", "US30Y_Rate_ret_20d", "IWM_SmallCap_vol_20d", "EWM_Malaysia_vol_20d", "NEE_NextEra_ret_20d", "ASX_Australia_ret_5d", "TED_Spread_zscore_60d", "SCHW_Schwab_ret_5d", "TGT_Target_zscore_60d", "AVB_AvalonBay_zscore_60d", "ORCL_zscore_60d", "DE_Deere_vol_20d", "3M_ret_5d", "DHR_ret_1d", "heston_var_ev_h5", "EWM_Malaysia_ret_1d", "EWJ_Japan_vol_20d", "T_ret_1d"], "is_new": true}, {"model_id": "new_h2_GLOBAL_GradientBoosting_N30_t6", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 2, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "AMD_ret_1d", "US5Y_Rate_ret_5d", "XLY_Disc_vol_20d", "Nikkei_Japan_vol_20d", "MS_MorganStanley_zscore_60d", "Retail_Sales_zscore_60d", "spx_momentum_3d", "EWL_Switzerland_zscore_60d", "PAYX_Paychex_ret_20d", "EMR_Emerson_ret_20d", "AMGN_Amgen_ret_1d", "EWG_Germany_ret_20d", "heston_var_ev_h3", "EWM_Malaysia_ret_1d", "HUM_Humana_ret_5d", "EWY_Korea_zscore_60d", "SJM_JM_Smucker_ret_5d", "spx_vol_5d", "MSTR_Bitcoin3_ret_1d", "LOW_Lowes_ret_5d", "NVDA_vol_20d", "SBUX_ret_5d", "3M_ret_5d", "PFE_ret_1d", "CPB_CampbellSoup_ret_5d", "US7Y_Rate_ret_20d", "LLY_zscore_60d", "Michigan_Sentiment_ret_20d"], "is_new": true}, {"model_id": "new_h2_GLOBAL_GradientBoosting_N30_t7", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 2, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWA_Australia_zscore_60d", "VOD_Vodafone_zscore_60d", "NEE_NextEra_ret_20d", "SBUX_zscore_60d", "Retail_Sales_zscore_60d", "ASX_Australia_ret_5d", "INTC_ret_5d", "EXC_Exelon_zscore_60d", "MS_MorganStanley_ret_5d", "PAYX_Paychex_zscore_60d", "EWG_Germany_ret_20d", "WTI_Oil_FRED_zscore_60d", "EWS_Singapore_ret_5d", "Brent_Oil_FRED_ret_5d", "heston_var_ev_h3", "EWY_Korea_zscore_60d", "VVIX_ret_20d", "3M_vol_20d", "VRP_ma5", "JNJ_ret_1d", "GE_ret_1d", "NWL_Newell_ret_20d", "T_ret_1d", "TGT_Target_zscore_60d", "SCHW_Schwab_ret_5d", "EWQ_France_zscore_60d", "AXP_Amex_vol_20d", "LMT_LockheedMartin_vol_20d"], "is_new": true}, {"model_id": "new_h2_GLOBAL_RandomForest_N5_t0", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 2, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EMR_Emerson_ret_20d", "IYM_BasicMaterials_ret_20d", "EWJ_Japan_vol_20d"], "is_new": true}, {"model_id": "new_h2_GLOBAL_RandomForest_N5_t1", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 2, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "NVDA_vol_20d", "DAX_Germany_zscore_60d", "INTC_ret_1d"], "is_new": true}, {"model_id": "new_h2_GLOBAL_RandomForest_N5_t2", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 2, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "US3M_Rate_vol_20d", "IWM_SmallCap_vol_20d", "SPY_zscore_60d"], "is_new": true}, {"model_id": "new_h2_GLOBAL_RandomForest_N5_t3", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 2, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "HUM_Humana_ret_5d", "MSTR_Bitcoin3_ret_5d", "DAX_Germany_vol_20d"], "is_new": true}, {"model_id": "new_h2_GLOBAL_RandomForest_N5_t4", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 2, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "AMZN_ret_5d", "Brent_Oil_FRED_ret_20d", "US6M_Rate_ret_20d"], "is_new": true}, {"model_id": "new_h2_GLOBAL_RandomForest_N5_t5", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 2, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWH_HongKong_ret_5d", "SPY_zscore_60d", "SLB_Schlumberger_ret_1d"], "is_new": true}, {"model_id": "new_h2_GLOBAL_RandomForest_N5_t6", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 2, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWM_Malaysia_zscore_60d", "gjr_condvar_h1", "IBEX_Spain_ret_20d"], "is_new": true}, {"model_id": "new_h2_GLOBAL_RandomForest_N5_t7", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 2, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "M_Macys_vol_20d", "US3Y_Rate_ret_5d", "EWM_Malaysia_zscore_60d"], "is_new": true}, {"model_id": "new_h2_GLOBAL_RandomForest_N8_t0", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 2, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWH_HongKong_ret_5d", "CPB_CampbellSoup_zscore_60d", "US5Y_Rate_ret_5d", "TED_Spread_zscore_60d", "INTC_ret_5d", "MSTR_Bitcoin3_ret_1d"], "is_new": true}, {"model_id": "new_h2_GLOBAL_RandomForest_N8_t1", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 2, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "CPB_CampbellSoup_zscore_60d", "EWG_Germany_ret_20d", "AMD_ret_5d", "LMT_LockheedMartin_ret_1d", "HUM_Humana_ret_5d", "DAX_Germany_zscore_60d"], "is_new": true}, {"model_id": "new_h2_GLOBAL_RandomForest_N8_t2", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 2, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "gjr_condvar_h1", "BTI_BritishAmerican_ret_5d", "CPB_CampbellSoup_vol_20d", "CLX_Clorox_vol_20d", "US7Y_Rate_ret_20d", "spx_abs_ret_max_5d"], "is_new": true}, {"model_id": "new_h2_GLOBAL_RandomForest_N8_t3", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 2, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "AMD_ret_5d", "US5Y_Rate_ret_5d", "HUM_Humana_ret_5d", "ENB_EnbridgeInc_ret_1d", "spx_vol_5d", "HD_ret_20d"], "is_new": true}, {"model_id": "new_h2_GLOBAL_RandomForest_N8_t4", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 2, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "XOM_ret_1d", "VOD_Vodafone_zscore_60d", "ASX_Australia_vol_20d", "EWM_Malaysia_vol_20d", "DHR_ret_1d", "EWY_Korea_ret_20d"], "is_new": true}, {"model_id": "new_h2_GLOBAL_RandomForest_N8_t5", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 2, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "MS_MorganStanley_ret_1d", "Nikkei_Japan_zscore_60d", "ASX_Australia_vol_20d", "gjr_condvar_h1", "ENB_EnbridgeInc_ret_1d", "TED_Spread_vol_20d"], "is_new": true}, {"model_id": "new_h2_GLOBAL_RandomForest_N8_t6", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 2, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "IBEX_Spain_ret_20d", "AMGN_Amgen_ret_1d", "heston_var_ev_h7", "EWJ_Japan_vol_20d", "US3M_Rate_vol_20d", "XLK_Tech_zscore_60d"], "is_new": true}, {"model_id": "new_h2_GLOBAL_RandomForest_N8_t7", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 2, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "HD_zscore_60d", "vix_mean_abs_ret_5d", "spx_abs_ret_max_5d", "EWA_Australia_zscore_60d", "AMT_AmericanTower_ret_1d", "TED_Spread_zscore_60d"], "is_new": true}, {"model_id": "new_h2_GLOBAL_RandomForest_N10_t0", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 2, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EMR_Emerson_ret_20d", "US3M_Rate_zscore_60d", "vix_acceleration_1d", "EWG_Germany_vol_20d", "heston_ev_h3", "XOM_ret_1d", "PAYX_Paychex_vol_20d", "AMD_ret_1d"], "is_new": true}, {"model_id": "new_h2_GLOBAL_RandomForest_N10_t1", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 2, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "SLB_Schlumberger_ret_5d", "ENB_EnbridgeInc_ret_1d", "3M_vol_20d", "PPL_PPL_ret_1d", "DAX_Germany_vol_20d", "LMT_LockheedMartin_ret_1d", "IBEX_Spain_ret_20d", "US30Y_Rate_ret_20d"], "is_new": true}, {"model_id": "new_h2_GLOBAL_RandomForest_N10_t2", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 2, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWM_Malaysia_ret_1d", "CPB_CampbellSoup_vol_20d", "Brent_Oil_FRED_ret_5d", "PPL_PPL_ret_1d", "SLB_Schlumberger_ret_5d", "NOC_Northrop_ret_20d", "BDX_Becton_Dickinson_ret_20d", "JNJ_ret_1d"], "is_new": true}, {"model_id": "new_h2_GLOBAL_RandomForest_N10_t3", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 2, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "SCHW_Schwab_ret_5d", "Brent_Oil_FRED_ret_20d", "VRP_ma5", "DE_Deere_vol_20d", "vix_acceleration_1d", "US30Y_Rate_ret_20d", "US1Y_Rate_ret_20d", "XLF_Fin_vol_20d"], "is_new": true}, {"model_id": "new_h2_GLOBAL_RandomForest_N10_t4", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 2, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EFFR_ret_1d", "TED_Spread_zscore_60d", "PAYX_Paychex_ret_20d", "INTC_ret_5d", "ORCL_zscore_60d", "GE_ret_1d", "ASX_Australia_ret_5d", "LOW_Lowes_ret_20d"], "is_new": true}, {"model_id": "new_h2_GLOBAL_RandomForest_N10_t5", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 2, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWH_HongKong_ret_5d", "NOC_Northrop_ret_20d", "AMD_ret_5d", "MSTR_Bitcoin3_ret_1d", "ORCL_zscore_60d", "SBUX_zscore_60d", "TED_Spread_zscore_60d", "SBUX_ret_5d"], "is_new": true}, {"model_id": "new_h2_GLOBAL_RandomForest_N10_t6", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 2, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "spx_vol_5d", "EOG_EOGResources_vol_20d", "MSTR_Bitcoin3_ret_1d", "ITT_ITTInc_ret_5d", "JNJ_ret_1d", "PLD_Prologis_ret_5d", "BTI_BritishAmerican_ret_20d", "EWA_Australia_ret_1d"], "is_new": true}, {"model_id": "new_h2_GLOBAL_RandomForest_N10_t7", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 2, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EXC_Exelon_zscore_60d", "LUV_SouthwestAir_ret_5d", "CPB_CampbellSoup_ret_20d", "IBEX_Spain_ret_20d", "INTC_ret_1d", "EWM_Malaysia_vol_20d", "PAYX_Paychex_zscore_60d", "LOW_Lowes_ret_20d"], "is_new": true}, {"model_id": "new_h2_GLOBAL_RandomForest_N12_t0", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 2, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "HangSeng_HK_ret_1d", "SPY_zscore_60d", "TGT_Target_zscore_60d", "DIS_vol_20d", "ITT_ITTInc_ret_5d", "DHR_ret_1d", "NVDA_vol_20d", "spx_abs_ret_max_5d", "XLB_Materials_zscore_60d", "US3Y_Rate_ret_5d"], "is_new": true}, {"model_id": "new_h2_GLOBAL_RandomForest_N12_t1", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 2, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "CPB_CampbellSoup_vol_20d", "CTAS_Cintas_vol_20d", "INTC_ret_1d", "Nikkei_Japan_zscore_60d", "QQQ_vol_20d", "EWY_Korea_ret_20d", "EWM_Malaysia_zscore_60d", "TXN_vol_20d", "LLY_zscore_60d", "Brent_Oil_FRED_ret_5d"], "is_new": true}, {"model_id": "new_h2_GLOBAL_RandomForest_N12_t2", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 2, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "NVDA_vol_20d", "SCHW_Schwab_ret_5d", "ASX_Australia_vol_20d", "hmm_p_stress", "MO_AltriaMG_ret_1d", "BLK_BlackRock_zscore_60d", "spx_vol_5d", "NFCI_ret_5d", "TGT_Target_zscore_60d", "AMD_ret_1d"], "is_new": true}, {"model_id": "new_h2_GLOBAL_RandomForest_N12_t3", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 2, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "US7Y_Rate_ret_20d", "EWM_Malaysia_ret_1d", "EXC_Exelon_zscore_60d", "ORCL_vol_20d", "ITT_ITTInc_ret_5d", "BLK_BlackRock_zscore_60d", "EQIX_Equinix_ret_5d", "SBUX_vol_20d", "INTC_ret_5d", "SLB_Schlumberger_ret_5d"], "is_new": true}, {"model_id": "new_h2_GLOBAL_RandomForest_N12_t4", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 2, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "LLY_zscore_60d", "MSTR_Bitcoin3_ret_1d", "EWG_Germany_ret_20d", "EWQ_France_zscore_60d", "EWG_Germany_vol_20d", "TM_Telephone_vol_20d", "US5Y_Rate_ret_5d", "DOW_Price_zscore_60d", "EWS_Singapore_ret_5d", "HangSeng_HK_ret_1d"], "is_new": true}, {"model_id": "new_h2_GLOBAL_RandomForest_N12_t5", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 2, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "ITT_ITTInc_ret_5d", "PAYX_Paychex_zscore_60d", "BA_ret_1d", "Industrial_Production_zscore_60d", "Brent_Oil_FRED_ret_5d", "INTC_ret_5d", "PPL_PPL_ret_1d", "TED_Spread_zscore_60d", "heston_var_ev_h3", "DE_Deere_vol_20d"], "is_new": true}, {"model_id": "new_h2_GLOBAL_RandomForest_N12_t6", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 2, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "ENB_EnbridgeInc_ret_1d", "GD_GeneralDynamics_zscore_60d", "DIS_vol_20d", "PG_ret_20d", "CLX_Clorox_vol_20d", "MSTR_Bitcoin3_ret_1d", "CTAS_Cintas_vol_20d", "vix_acceleration_1d", "TED_Spread_vol_20d", "vix_mean_abs_ret_5d"], "is_new": true}, {"model_id": "new_h2_GLOBAL_RandomForest_N12_t7", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 2, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWY_Korea_zscore_60d", "SBUX_ret_5d", "NOC_Northrop_ret_20d", "EQR_Equity_ret_1d", "EWL_Switzerland_vol_20d", "gjr_condvar_h1", "PPL_PPL_ret_1d", "AMZN_ret_5d", "EWH_HongKong_ret_5d", "US1Y_Rate_ret_5d"], "is_new": true}, {"model_id": "new_h2_GLOBAL_RandomForest_N15_t0", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 2, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "ES_Evergy_ret_1d", "NFCI_ret_5d", "TED_Spread_vol_20d", "CPB_CampbellSoup_ret_20d", "US3Y_Rate_ret_5d", "EWA_Australia_ret_1d", "DAX_Germany_vol_20d", "XLK_Tech_zscore_60d", "EFFR_ret_1d", "EQIX_Equinix_ret_5d", "HD_ret_1d", "CMCSA_ret_1d", "EOG_EOGResources_vol_20d"], "is_new": true}, {"model_id": "new_h2_GLOBAL_RandomForest_N15_t1", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 2, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "MS_MorganStanley_ret_5d", "ES_Evergy_ret_1d", "LMT_LockheedMartin_ret_1d", "EWQ_France_zscore_60d", "US6M_Rate_ret_20d", "EWY_Korea_zscore_60d", "CPB_CampbellSoup_ret_20d", "SJM_JM_Smucker_ret_5d", "LOW_Lowes_ret_5d", "GILD_Gilead_ret_20d", "JNJ_ret_1d", "MO_AltriaMG_ret_1d", "HD_ret_1d"], "is_new": true}, {"model_id": "new_h2_GLOBAL_RandomForest_N15_t2", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 2, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "CI_Cigna_vol_20d", "EXC_Exelon_zscore_60d", "3M_ret_5d", "BA_ret_1d", "EWM_Malaysia_zscore_60d", "LLY_zscore_60d", "M_Macys_vol_20d", "EWM_Malaysia_ret_1d", "AORD_AUS_zscore_60d", "EWG_Germany_ret_20d", "EFFR_vol_20d", "SLB_Schlumberger_ret_1d", "DOW_Price_zscore_60d"], "is_new": true}, {"model_id": "new_h2_GLOBAL_RandomForest_N15_t3", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 2, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "vix_acceleration_1d", "HangSeng_HK_ret_5d", "XLK_Tech_zscore_60d", "AMD_ret_1d", "HangSeng_HK_vol_20d", "PAYX_Paychex_vol_20d", "US6M_Rate_ret_20d", "TXN_vol_20d", "MSTR_Bitcoin3_ret_20d", "LOW_Lowes_ret_5d", "PAYX_Paychex_ret_20d", "CPB_CampbellSoup_ret_5d", "Nikkei_Japan_vol_20d"], "is_new": true}, {"model_id": "new_h2_GLOBAL_RandomForest_N15_t4", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 2, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWA_Australia_zscore_60d", "DIS_vol_20d", "LUV_SouthwestAir_ret_5d", "BLK_BlackRock_zscore_60d", "TED_Spread_vol_20d", "NOC_Northrop_ret_20d", "EOG_EOGResources_vol_20d", "TM_Telephone_vol_20d", "NEE_NextEra_ret_20d", "TM_Telephone_ret_1d", "INTC_ret_1d", "MSTR_Bitcoin3_ret_5d", "HD_zscore_60d"], "is_new": true}, {"model_id": "new_h2_GLOBAL_RandomForest_N15_t5", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 2, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "MSTR_Bitcoin3_ret_5d", "AXP_Amex_vol_20d", "TED_Spread_vol_20d", "TM_Telephone_ret_1d", "EOG_EOGResources_vol_20d", "EWH_HongKong_ret_5d", "AMT_AmericanTower_ret_1d", "US5Y_Rate_ret_5d", "EOG_EOGResources_ret_5d", "LOW_Lowes_ret_5d", "TXN_vol_20d", "ORCL_zscore_60d", "EQIX_Equinix_ret_5d"], "is_new": true}, {"model_id": "new_h2_GLOBAL_RandomForest_N15_t6", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 2, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "vix_acceleration_1d", "EWJ_Japan_vol_20d", "MS_MorganStanley_ret_5d", "EWH_HongKong_ret_5d", "Brent_Oil_FRED_ret_5d", "MS_MorganStanley_zscore_60d", "EWG_Germany_ret_20d", "EWS_Singapore_ret_5d", "AMGN_Amgen_ret_1d", "US1Y_Rate_ret_20d", "HUM_Humana_ret_5d", "MS_MorganStanley_ret_1d", "AVB_AvalonBay_zscore_60d"], "is_new": true}, {"model_id": "new_h2_GLOBAL_RandomForest_N15_t7", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 2, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "DOW_Price_zscore_60d", "SO_SouthernCo_ret_5d", "AMD_ret_5d", "EWQ_France_ret_20d", "EWH_HongKong_ret_5d", "Michigan_Sentiment_ret_20d", "IBEX_Spain_ret_20d", "XLF_Fin_vol_20d", "AXP_Amex_vol_20d", "HD_ret_20d", "vix_acceleration_1d", "SJM_JM_Smucker_ret_1d", "US3Y_Rate_ret_5d"], "is_new": true}, {"model_id": "new_h2_GLOBAL_RandomForest_N20_t0", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 2, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EXC_Exelon_ret_1d", "hmm_p_stress", "EWL_Switzerland_zscore_60d", "ASX_Australia_ret_5d", "EWM_Malaysia_vol_20d", "CPB_CampbellSoup_ret_20d", "BLK_BlackRock_zscore_60d", "CPB_CampbellSoup_ret_5d", "MS_MorganStanley_ret_1d", "NFCI_ret_5d", "EWQ_France_ret_20d", "MSTR_Bitcoin3_ret_1d", "Nikkei_Japan_zscore_60d", "EQIX_Equinix_ret_5d", "DHR_vol_20d", "TED_Spread_vol_20d", "GILD_Gilead_ret_20d", "LOW_Lowes_ret_5d"], "is_new": true}, {"model_id": "new_h2_GLOBAL_RandomForest_N20_t1", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 2, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "3M_ret_5d", "TGT_Target_zscore_60d", "heston_ev_h3", "IYM_BasicMaterials_ret_20d", "PCAR_PaccarInc_ret_5d", "XLV_Health_zscore_60d", "MSTR_Bitcoin3_ret_1d", "DOW_Price_zscore_60d", "EMR_Emerson_ret_20d", "US5Y_Rate_ret_5d", "LMT_LockheedMartin_vol_20d", "Brent_Oil_FRED_ret_5d", "CPB_CampbellSoup_vol_20d", "SLB_Schlumberger_ret_5d", "IBEX_Spain_ret_20d", "TED_Spread_vol_20d", "Nikkei_Japan_vol_20d", "CCI_CrownCastle_vol_20d"], "is_new": true}, {"model_id": "new_h2_GLOBAL_RandomForest_N20_t2", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 2, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "spx_momentum_3d", "BLK_BlackRock_zscore_60d", "EQR_Equity_ret_1d", "MS_MorganStanley_ret_1d", "3M_vol_20d", "HUM_Humana_ret_5d", "NVDA_vol_20d", "VRP_ma5", "AMGN_Amgen_ret_1d", "heston_var_ev_h3", "CPB_CampbellSoup_vol_20d", "PAYX_Paychex_ret_20d", "FedFunds_zscore_60d", "XLK_Tech_zscore_60d", "IYR_US_REIT2_zscore_60d", "EWG_Germany_ret_20d", "SLB_Schlumberger_ret_5d", "heston_ev_h3"], "is_new": true}, {"model_id": "new_h2_GLOBAL_RandomForest_N20_t3", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 2, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "IBEX_Spain_ret_20d", "HD_ret_1d", "Brent_Oil_FRED_ret_20d", "AMZN_ret_5d", "NOC_Northrop_ret_20d", "MS_MorganStanley_ret_1d", "XOM_ret_20d", "SBUX_zscore_60d", "TXN_vol_20d", "HD_ret_5d", "INTC_ret_5d", "PPL_PPL_ret_1d", "EWL_Switzerland_zscore_60d", "DIS_vol_20d", "Brent_Oil_FRED_ret_5d", "Nikkei_Japan_zscore_60d", "LMT_LockheedMartin_ret_1d", "BDX_Becton_Dickinson_ret_20d"], "is_new": true}, {"model_id": "new_h2_GLOBAL_RandomForest_N20_t4", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 2, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "TED_Spread_vol_20d", "US3Y_Rate_ret_5d", "EWA_Australia_zscore_60d", "LLY_zscore_60d", "GD_GeneralDynamics_zscore_60d", "EXC_Exelon_ret_1d", "AMD_ret_5d", "PFE_ret_1d", "Brent_Oil_FRED_ret_5d", "BTI_BritishAmerican_ret_5d", "heston_var_ev_h5", "PG_ret_20d", "EOG_EOGResources_ret_5d", "XLV_Health_zscore_60d", "SPY_zscore_60d", "MRK_Merck_zscore_60d", "AMZN_ret_5d", "EWS_Singapore_ret_5d"], "is_new": true}, {"model_id": "new_h2_GLOBAL_RandomForest_N20_t5", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 2, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "NWL_Newell_ret_20d", "ORCL_vol_20d", "PAYX_Paychex_ret_20d", "CPB_CampbellSoup_ret_20d", "EWY_Korea_zscore_60d", "TM_Telephone_vol_20d", "Industrial_Production_zscore_60d", "EWM_Malaysia_zscore_60d", "EWL_Switzerland_zscore_60d", "spx_momentum_3d", "EWC_Canada_zscore_60d", "DAX_Germany_vol_20d", "SJM_JM_Smucker_ret_1d", "CCI_CrownCastle_vol_20d", "EXC_Exelon_ret_1d", "PCAR_PaccarInc_ret_5d", "US5Y_Rate_ret_5d", "INTC_ret_5d"], "is_new": true}, {"model_id": "new_h2_GLOBAL_RandomForest_N20_t6", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 2, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "US3M_Rate_vol_20d", "US1Y_Rate_ret_20d", "ENB_EnbridgeInc_ret_1d", "HD_zscore_60d", "US6M_Rate_ret_20d", "EQIX_Equinix_ret_5d", "EFFR_vol_20d", "3M_vol_20d", "EQR_Equity_ret_1d", "SLB_Schlumberger_ret_1d", "vix_mean_abs_ret_5d", "T_ret_1d", "Brent_Oil_FRED_ret_20d", "LOW_Lowes_ret_20d", "spx_abs_ret_max_5d", "AXP_Amex_vol_20d", "SLB_Schlumberger_ret_5d", "VOD_Vodafone_zscore_60d"], "is_new": true}, {"model_id": "new_h2_GLOBAL_RandomForest_N20_t7", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 2, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "ORCL_zscore_60d", "MS_MorganStanley_ret_5d", "T_ret_1d", "3M_ret_5d", "Core_PCE_zscore_60d", "EFFR_vol_20d", "NOC_Northrop_ret_20d", "XLF_Fin_vol_20d", "Nikkei_Japan_zscore_60d", "ENB_EnbridgeInc_ret_1d", "IYR_US_REIT2_zscore_60d", "AXP_Amex_ret_20d", "IWM_SmallCap_vol_20d", "BDX_Becton_Dickinson_ret_20d", "Retail_Sales_zscore_60d", "EWM_Malaysia_vol_20d", "DE_Deere_vol_20d", "HD_ret_1d"], "is_new": true}, {"model_id": "new_h2_GLOBAL_RandomForest_N25_t0", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 2, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWQ_France_zscore_60d", "BA_ret_1d", "EWY_Korea_ret_20d", "EOG_EOGResources_ret_5d", "LUV_SouthwestAir_ret_5d", "AXP_Amex_ret_20d", "spx_momentum_3d", "SBUX_vol_20d", "EWY_Korea_zscore_60d", "WTI_Oil_FRED_zscore_60d", "MS_MorganStanley_zscore_60d", "DOW_Price_zscore_60d", "vix_mean_abs_ret_5d", "US30Y_Rate_ret_20d", "EWM_Malaysia_vol_20d", "SLB_Schlumberger_ret_1d", "HD_ret_20d", "HUM_Humana_ret_5d", "AXP_Amex_vol_20d", "EWM_Malaysia_zscore_60d", "SBUX_ret_5d", "MS_MorganStanley_ret_5d", "LMT_LockheedMartin_vol_20d"], "is_new": true}, {"model_id": "new_h2_GLOBAL_RandomForest_N25_t1", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 2, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "US30Y_Rate_ret_20d", "LOW_Lowes_ret_20d", "TED_Spread_vol_20d", "AMGN_Amgen_ret_1d", "EWA_Australia_ret_1d", "EWM_Malaysia_ret_1d", "AMZN_ret_5d", "T10Y2Y_Spread_ret_5d", "NFCI_ret_5d", "EXC_Exelon_ret_1d", "CPB_CampbellSoup_ret_20d", "HD_ret_5d", "US1Y_Rate_ret_5d", "heston_var_ev_h3", "EWS_Singapore_ret_5d", "DAX_Germany_zscore_60d", "VRP_ma5", "AXP_Amex_ret_20d", "SO_SouthernCo_ret_5d", "INTC_ret_5d", "AMT_AmericanTower_ret_1d", "IYR_US_REIT2_zscore_60d", "BDX_Becton_Dickinson_ret_20d"], "is_new": true}, {"model_id": "new_h2_GLOBAL_RandomForest_N25_t2", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 2, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "US7Y_Rate_ret_20d", "AORD_AUS_zscore_60d", "INTC_ret_1d", "Core_PCE_zscore_60d", "US1Y_Rate_ret_5d", "DAX_Germany_zscore_60d", "EWJ_Japan_vol_20d", "PLD_Prologis_ret_5d", "SBUX_vol_20d", "EOG_EOGResources_ret_5d", "CPB_CampbellSoup_vol_20d", "BLK_BlackRock_zscore_60d", "XLV_Health_zscore_60d", "BA_ret_1d", "SBUX_ret_5d", "EWY_Korea_zscore_60d", "HD_ret_5d", "EXC_Exelon_ret_1d", "PPL_PPL_ret_1d", "US3Y_Rate_ret_5d", "EWM_Malaysia_zscore_60d", "US3M_Rate_zscore_60d", "CPB_CampbellSoup_zscore_60d"], "is_new": true}, {"model_id": "new_h2_GLOBAL_RandomForest_N25_t3", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 2, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "SBUX_vol_20d", "EWY_Korea_zscore_60d", "AMD_ret_1d", "AORD_AUS_zscore_60d", "EWG_Germany_ret_20d", "EXC_Exelon_ret_1d", "LOW_Lowes_ret_20d", "MS_MorganStanley_ret_5d", "TED_Spread_zscore_60d", "SLB_Schlumberger_ret_1d", "GILD_Gilead_ret_20d", "heston_var_ev_h3", "MS_MorganStanley_zscore_60d", "IYM_BasicMaterials_ret_20d", "vix_mean_abs_ret_5d", "EWH_HongKong_ret_5d", "Retail_Sales_zscore_60d", "HUM_Humana_ret_5d", "HD_ret_1d", "MSTR_Bitcoin3_ret_1d", "Michigan_Sentiment_ret_20d", "VVIX_ret_20d", "3M_vol_20d"], "is_new": true}, {"model_id": "new_h2_GLOBAL_RandomForest_N25_t4", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 2, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "MSTR_Bitcoin3_ret_1d", "BDX_Becton_Dickinson_ret_20d", "HD_zscore_60d", "PPL_PPL_ret_1d", "FedFunds_zscore_60d", "CPB_CampbellSoup_vol_20d", "DIS_vol_20d", "3M_vol_20d", "US1Y_Rate_ret_5d", "CPB_CampbellSoup_zscore_60d", "TXN_vol_20d", "3M_ret_5d", "EWA_Australia_zscore_60d", "ORCL_vol_20d", "SPY_zscore_60d", "VVIX_ret_20d", "EWH_HongKong_ret_5d", "SJM_JM_Smucker_ret_5d", "EFFR_ret_1d", "LOW_Lowes_ret_20d", "DE_Deere_ret_5d", "TGT_Target_zscore_60d", "US7Y_Rate_ret_20d"], "is_new": true}, {"model_id": "new_h2_GLOBAL_RandomForest_N25_t5", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 2, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWL_Switzerland_vol_20d", "US6M_Rate_ret_20d", "EWY_Korea_ret_20d", "LUV_SouthwestAir_ret_5d", "Core_CPI_zscore_60d", "M_Macys_vol_20d", "AMGN_Amgen_ret_1d", "DHR_ret_1d", "MS_MorganStanley_zscore_60d", "XLF_Fin_vol_20d", "IYM_BasicMaterials_ret_20d", "SBUX_ret_5d", "ORCL_zscore_60d", "MRK_Merck_zscore_60d", "DE_Deere_ret_5d", "Nikkei_Japan_zscore_60d", "NVDA_vol_20d", "US30Y_Rate_ret_20d", "HD_zscore_60d", "XLB_Materials_zscore_60d", "BA_ret_1d", "vix_mean_abs_ret_5d", "CPB_CampbellSoup_ret_5d"], "is_new": true}, {"model_id": "new_h2_GLOBAL_RandomForest_N25_t6", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 2, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "T10Y2Y_Spread_ret_5d", "BTI_BritishAmerican_ret_5d", "PFE_ret_1d", "INTC_ret_1d", "DOW_Price_zscore_60d", "EXC_Exelon_zscore_60d", "heston_var_ev_h7", "US1Y_Rate_ret_20d", "vix_acceleration_1d", "EWY_Korea_ret_20d", "Industrial_Production_zscore_60d", "EWS_Singapore_ret_5d", "SBUX_vol_20d", "MSTR_Bitcoin3_ret_5d", "GD_GeneralDynamics_zscore_60d", "EFFR_vol_20d", "SO_SouthernCo_ret_5d", "EWM_Malaysia_vol_20d", "EWL_Switzerland_zscore_60d", "AMD_ret_1d", "EWH_HongKong_ret_5d", "LOW_Lowes_ret_20d", "EWQ_France_zscore_60d"], "is_new": true}, {"model_id": "new_h2_GLOBAL_RandomForest_N25_t7", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 2, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "AVB_AvalonBay_zscore_60d", "T10Y2Y_Spread_ret_5d", "spx_abs_ret_max_5d", "TM_Telephone_ret_1d", "LLY_zscore_60d", "EWA_Australia_ret_1d", "EQR_Equity_ret_1d", "WTI_Oil_FRED_zscore_60d", "US30Y_Rate_ret_20d", "PLD_Prologis_ret_5d", "EWQ_France_ret_20d", "HD_zscore_60d", "AORD_AUS_zscore_60d", "CMCSA_ret_1d", "CI_Cigna_vol_20d", "EXC_Exelon_ret_1d", "XLB_Materials_zscore_60d", "SBUX_zscore_60d", "HangSeng_HK_ret_5d", "PCAR_PaccarInc_ret_5d", "IYM_BasicMaterials_ret_20d", "GE_ret_1d", "XLF_Fin_vol_20d"], "is_new": true}, {"model_id": "new_h2_GLOBAL_RandomForest_N30_t0", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 2, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "GILD_Gilead_ret_20d", "VOD_Vodafone_zscore_60d", "XLV_Health_zscore_60d", "AVB_AvalonBay_zscore_60d", "US5Y_Rate_ret_5d", "NOC_Northrop_ret_20d", "PFE_ret_1d", "TGT_Target_zscore_60d", "BTI_BritishAmerican_ret_5d", "XLB_Materials_zscore_60d", "IBEX_Spain_ret_20d", "LUV_SouthwestAir_ret_5d", "MS_MorganStanley_ret_1d", "Industrial_Production_zscore_60d", "spx_abs_ret_max_5d", "Retail_Sales_zscore_60d", "PAYX_Paychex_zscore_60d", "EWM_Malaysia_zscore_60d", "MSTR_Bitcoin3_ret_5d", "CPB_CampbellSoup_ret_20d", "ASX_Australia_ret_5d", "EWG_Germany_ret_20d", "EWY_Korea_zscore_60d", "AMT_AmericanTower_ret_1d", "JNJ_ret_1d", "PPL_PPL_ret_1d", "EWG_Germany_vol_20d", "M_Macys_vol_20d"], "is_new": true}, {"model_id": "new_h2_GLOBAL_RandomForest_N30_t1", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 2, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "HUM_Humana_ret_5d", "CTAS_Cintas_vol_20d", "SJM_JM_Smucker_ret_1d", "INTC_ret_1d", "Nikkei_Japan_vol_20d", "XOM_ret_1d", "heston_var_ev_h3", "HD_ret_1d", "EWJ_Japan_vol_20d", "ASX_Australia_ret_5d", "QQQ_vol_20d", "EMR_Emerson_ret_20d", "CPB_CampbellSoup_vol_20d", "SJM_JM_Smucker_ret_5d", "SLB_Schlumberger_ret_1d", "PCAR_PaccarInc_ret_5d", "XLK_Tech_zscore_60d", "BTI_BritishAmerican_ret_5d", "NWL_Newell_ret_20d", "US5Y_Rate_ret_5d", "VOD_Vodafone_zscore_60d", "CPB_CampbellSoup_ret_5d", "MS_MorganStanley_ret_1d", "INTC_ret_5d", "XLV_Health_zscore_60d", "PG_ret_20d", "EQR_Equity_ret_1d", "EWM_Malaysia_zscore_60d"], "is_new": true}, {"model_id": "new_h2_GLOBAL_RandomForest_N30_t2", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 2, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "Core_PCE_zscore_60d", "ASX_Australia_ret_5d", "FedFunds_zscore_60d", "TM_Telephone_ret_1d", "IYM_BasicMaterials_ret_20d", "ORCL_zscore_60d", "US1Y_Rate_ret_5d", "NVDA_vol_20d", "PAYX_Paychex_zscore_60d", "CTAS_Cintas_vol_20d", "DIS_vol_20d", "MSTR_Bitcoin3_ret_5d", "IWM_SmallCap_vol_20d", "Retail_Sales_zscore_60d", "EFFR_vol_20d", "T_ret_1d", "BTI_BritishAmerican_ret_5d", "US1Y_Rate_ret_20d", "DHR_vol_20d", "VOD_Vodafone_zscore_60d", "vix_mean_abs_ret_5d", "ENB_EnbridgeInc_ret_1d", "SBUX_zscore_60d", "US30Y_Rate_ret_20d", "TGT_Target_zscore_60d", "SLB_Schlumberger_ret_5d", "LUV_SouthwestAir_ret_5d", "VVIX_ret_20d"], "is_new": true}, {"model_id": "new_h2_GLOBAL_RandomForest_N30_t3", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 2, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "heston_var_ev_h3", "TED_Spread_zscore_60d", "HD_zscore_60d", "CPB_CampbellSoup_ret_5d", "LMT_LockheedMartin_ret_1d", "US30Y_Rate_ret_20d", "IYR_US_REIT2_zscore_60d", "MO_AltriaMG_ret_1d", "EWA_Australia_zscore_60d", "PLD_Prologis_ret_5d", "EWM_Malaysia_vol_20d", "BLK_BlackRock_zscore_60d", "HD_ret_1d", "EWJ_Japan_vol_20d", "PCAR_PaccarInc_ret_5d", "NOC_Northrop_ret_20d", "EWG_Germany_vol_20d", "DAX_Germany_vol_20d", "vix_acceleration_1d", "ITT_ITTInc_ret_5d", "heston_var_ev_h5", "US3M_Rate_vol_20d", "Michigan_Sentiment_ret_20d", "SCHW_Schwab_ret_5d", "AMT_AmericanTower_ret_1d", "US6M_Rate_ret_20d", "XLB_Materials_zscore_60d", "ORCL_vol_20d"], "is_new": true}, {"model_id": "new_h2_GLOBAL_RandomForest_N30_t4", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 2, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "GE_ret_1d", "HangSeng_HK_vol_20d", "3M_ret_5d", "3M_vol_20d", "BLK_BlackRock_zscore_60d", "vix_mean_abs_ret_5d", "ASX_Australia_vol_20d", "PAYX_Paychex_ret_20d", "MO_AltriaMG_ret_1d", "NFCI_ret_5d", "AXP_Amex_ret_20d", "DOW_Price_zscore_60d", "EWQ_France_zscore_60d", "DAX_Germany_vol_20d", "BA_ret_1d", "NWL_Newell_ret_20d", "IBEX_Spain_ret_20d", "AVB_AvalonBay_zscore_60d", "heston_var_ev_h5", "SBUX_vol_20d", "EFFR_vol_20d", "US7Y_Rate_ret_20d", "heston_ev_h3", "TED_Spread_vol_20d", "EMR_Emerson_ret_20d", "EQIX_Equinix_ret_5d", "US30Y_Rate_ret_20d", "SJM_JM_Smucker_ret_5d"], "is_new": true}, {"model_id": "new_h2_GLOBAL_RandomForest_N30_t5", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 2, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "PCAR_PaccarInc_ret_5d", "EWG_Germany_vol_20d", "GD_GeneralDynamics_zscore_60d", "EXC_Exelon_zscore_60d", "VOD_Vodafone_zscore_60d", "SBUX_vol_20d", "US5Y_Rate_ret_5d", "WTI_Oil_FRED_zscore_60d", "DOW_Price_zscore_60d", "MRK_Merck_zscore_60d", "Core_CPI_zscore_60d", "VVIX_ret_20d", "Retail_Sales_zscore_60d", "EWA_Australia_zscore_60d", "NFCI_ret_5d", "NEE_NextEra_ret_20d", "ITT_ITTInc_ret_5d", "GILD_Gilead_ret_20d", "PAYX_Paychex_vol_20d", "LUV_SouthwestAir_ret_5d", "SLB_Schlumberger_ret_5d", "PG_ret_20d", "US30Y_Rate_ret_20d", "EWS_Singapore_ret_5d", "SBUX_ret_5d", "BTI_BritishAmerican_ret_5d", "BLK_BlackRock_zscore_60d", "XLF_Fin_vol_20d"], "is_new": true}, {"model_id": "new_h2_GLOBAL_RandomForest_N30_t6", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 2, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "Nikkei_Japan_zscore_60d", "SCHW_Schwab_ret_5d", "PFE_ret_1d", "Core_PCE_zscore_60d", "US1Y_Rate_ret_5d", "ITT_ITTInc_ret_5d", "EWA_Australia_zscore_60d", "M_Macys_vol_20d", "3M_ret_5d", "heston_ev_h3", "CPB_CampbellSoup_zscore_60d", "QQQ_vol_20d", "HD_zscore_60d", "EWM_Malaysia_ret_1d", "AMT_AmericanTower_ret_1d", "EXC_Exelon_ret_1d", "BLK_BlackRock_zscore_60d", "EFFR_vol_20d", "LMT_LockheedMartin_ret_1d", "MSTR_Bitcoin3_ret_1d", "XLF_Fin_vol_20d", "AXP_Amex_ret_20d", "HUM_Humana_ret_5d", "spx_momentum_3d", "EWG_Germany_ret_20d", "CPB_CampbellSoup_ret_5d", "NEE_NextEra_ret_20d", "MS_MorganStanley_zscore_60d"], "is_new": true}, {"model_id": "new_h2_GLOBAL_RandomForest_N30_t7", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 2, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "Retail_Sales_zscore_60d", "3M_ret_5d", "ORCL_zscore_60d", "CTAS_Cintas_vol_20d", "EFFR_ret_1d", "PAYX_Paychex_vol_20d", "US5Y_Rate_ret_5d", "PLD_Prologis_ret_5d", "CPB_CampbellSoup_vol_20d", "spx_vol_5d", "Nikkei_Japan_zscore_60d", "EWJ_Japan_vol_20d", "IWM_SmallCap_vol_20d", "AMD_ret_1d", "NFCI_ret_5d", "HUM_Humana_ret_5d", "EQIX_Equinix_ret_5d", "PFE_ret_1d", "EFFR_vol_20d", "US3M_Rate_zscore_60d", "AMZN_ret_5d", "XLK_Tech_zscore_60d", "EWH_HongKong_ret_5d", "SO_SouthernCo_ret_5d", "T_ret_1d", "CPB_CampbellSoup_ret_5d", "SBUX_vol_20d", "NEE_NextEra_ret_20d"], "is_new": true}, {"model_id": "new_h2_GLOBAL_LogisticRegression_N5_t0", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 2, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "ORCL_zscore_60d", "ASX_Australia_ret_5d", "DHR_vol_20d"], "is_new": true}, {"model_id": "new_h2_GLOBAL_LogisticRegression_N5_t1", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 2, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "Brent_Oil_FRED_ret_20d", "MS_MorganStanley_zscore_60d", "heston_var_ev_h3"], "is_new": true}, {"model_id": "new_h2_GLOBAL_LogisticRegression_N5_t2", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 2, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "NVDA_vol_20d", "GE_ret_1d", "SBUX_ret_5d"], "is_new": true}, {"model_id": "new_h2_GLOBAL_LogisticRegression_N5_t3", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 2, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "HD_ret_20d", "MSTR_Bitcoin3_ret_5d", "PLD_Prologis_ret_5d"], "is_new": true}, {"model_id": "new_h2_GLOBAL_LogisticRegression_N5_t4", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 2, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "ORCL_vol_20d", "XLY_Disc_vol_20d", "AMT_AmericanTower_ret_1d"], "is_new": true}, {"model_id": "new_h2_GLOBAL_LogisticRegression_N5_t5", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 2, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "ASX_Australia_ret_5d", "spx_vol_5d", "ES_Evergy_ret_1d"], "is_new": true}, {"model_id": "new_h2_GLOBAL_LogisticRegression_N5_t6", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 2, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "AORD_AUS_zscore_60d", "XOM_ret_1d", "PPL_PPL_ret_1d"], "is_new": true}, {"model_id": "new_h2_GLOBAL_LogisticRegression_N5_t7", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 2, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "BDX_Becton_Dickinson_ret_20d", "Nikkei_Japan_zscore_60d", "MSTR_Bitcoin3_ret_5d"], "is_new": true}, {"model_id": "new_h2_GLOBAL_LogisticRegression_N8_t0", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 2, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "HD_ret_5d", "spx_abs_ret_max_5d", "3M_vol_20d", "IWM_SmallCap_vol_20d", "EWM_Malaysia_zscore_60d", "heston_ev_h3"], "is_new": true}, {"model_id": "new_h2_GLOBAL_LogisticRegression_N8_t1", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 2, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "NFCI_ret_5d", "US3M_Rate_vol_20d", "HD_ret_1d", "EWL_Switzerland_zscore_60d", "IYR_US_REIT2_zscore_60d", "EWM_Malaysia_ret_1d"], "is_new": true}, {"model_id": "new_h2_GLOBAL_LogisticRegression_N8_t2", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 2, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "PLD_Prologis_ret_5d", "T_ret_1d", "vix_acceleration_1d", "AMD_ret_5d", "LUV_SouthwestAir_ret_5d", "US1Y_Rate_ret_20d"], "is_new": true}, {"model_id": "new_h2_GLOBAL_LogisticRegression_N8_t3", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 2, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "PAYX_Paychex_zscore_60d", "EWM_Malaysia_ret_1d", "ES_Evergy_ret_1d", "Core_CPI_zscore_60d", "MSTR_Bitcoin3_ret_1d", "PAYX_Paychex_ret_20d"], "is_new": true}, {"model_id": "new_h2_GLOBAL_LogisticRegression_N8_t4", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 2, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "IYR_US_REIT2_zscore_60d", "MRK_Merck_zscore_60d", "DE_Deere_vol_20d", "NEE_NextEra_ret_20d", "CTAS_Cintas_vol_20d", "INTC_ret_1d"], "is_new": true}, {"model_id": "new_h2_GLOBAL_LogisticRegression_N8_t5", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 2, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "LMT_LockheedMartin_ret_1d", "vix_acceleration_1d", "LUV_SouthwestAir_ret_5d", "SJM_JM_Smucker_ret_1d", "AMD_ret_1d", "EFFR_vol_20d"], "is_new": true}, {"model_id": "new_h2_GLOBAL_LogisticRegression_N8_t6", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 2, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "TGT_Target_zscore_60d", "SCHW_Schwab_ret_5d", "GD_GeneralDynamics_zscore_60d", "HD_zscore_60d", "CMCSA_ret_1d", "DAX_Germany_zscore_60d"], "is_new": true}, {"model_id": "new_h2_GLOBAL_LogisticRegression_N8_t7", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 2, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "INTC_ret_5d", "US30Y_Rate_ret_20d", "MS_MorganStanley_zscore_60d", "VVIX_ret_20d", "DOW_Price_zscore_60d", "M_Macys_vol_20d"], "is_new": true}, {"model_id": "new_h2_GLOBAL_LogisticRegression_N10_t0", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 2, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "ASX_Australia_ret_5d", "MSTR_Bitcoin3_ret_5d", "heston_var_ev_h7", "ES_Evergy_ret_1d", "IYR_US_REIT2_zscore_60d", "vix_acceleration_1d", "LLY_zscore_60d", "BTI_BritishAmerican_ret_20d"], "is_new": true}, {"model_id": "new_h2_GLOBAL_LogisticRegression_N10_t1", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 2, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "LOW_Lowes_ret_5d", "Brent_Oil_FRED_ret_5d", "US7Y_Rate_ret_20d", "EWG_Germany_ret_20d", "QQQ_vol_20d", "T10Y2Y_Spread_ret_5d", "EWY_Korea_zscore_60d", "VOD_Vodafone_zscore_60d"], "is_new": true}, {"model_id": "new_h2_GLOBAL_LogisticRegression_N10_t2", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 2, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "PCAR_PaccarInc_ret_5d", "EWC_Canada_zscore_60d", "GD_GeneralDynamics_zscore_60d", "EWY_Korea_zscore_60d", "IYR_US_REIT2_zscore_60d", "TED_Spread_zscore_60d", "heston_var_ev_h7", "AMT_AmericanTower_ret_1d"], "is_new": true}, {"model_id": "new_h2_GLOBAL_LogisticRegression_N10_t3", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 2, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EFFR_vol_20d", "CTAS_Cintas_vol_20d", "EWC_Canada_zscore_60d", "GD_GeneralDynamics_zscore_60d", "NOC_Northrop_ret_20d", "CLX_Clorox_vol_20d", "MS_MorganStanley_zscore_60d", "GILD_Gilead_ret_20d"], "is_new": true}, {"model_id": "new_h2_GLOBAL_LogisticRegression_N10_t4", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 2, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "DOW_Price_zscore_60d", "Nikkei_Japan_zscore_60d", "IYR_US_REIT2_zscore_60d", "EWY_Korea_ret_20d", "DHR_vol_20d", "DAX_Germany_zscore_60d", "EXC_Exelon_zscore_60d", "PAYX_Paychex_zscore_60d"], "is_new": true}, {"model_id": "new_h2_GLOBAL_LogisticRegression_N10_t5", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 2, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "GE_ret_1d", "TED_Spread_zscore_60d", "CPB_CampbellSoup_zscore_60d", "ENB_EnbridgeInc_ret_1d", "US7Y_Rate_ret_20d", "PAYX_Paychex_ret_20d", "Brent_Oil_FRED_ret_20d", "XLV_Health_zscore_60d"], "is_new": true}, {"model_id": "new_h2_GLOBAL_LogisticRegression_N10_t6", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 2, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWY_Korea_ret_20d", "PAYX_Paychex_vol_20d", "HangSeng_HK_ret_5d", "US1Y_Rate_ret_20d", "vix_acceleration_1d", "heston_var_ev_h7", "Core_PCE_zscore_60d", "DE_Deere_vol_20d"], "is_new": true}, {"model_id": "new_h2_GLOBAL_LogisticRegression_N10_t7", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 2, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "AMD_ret_1d", "AMD_ret_5d", "T10Y2Y_Spread_ret_5d", "AMGN_Amgen_ret_1d", "ITT_ITTInc_ret_5d", "SO_SouthernCo_ret_5d", "TM_Telephone_vol_20d", "VRP_ma5"], "is_new": true}, {"model_id": "new_h2_GLOBAL_LogisticRegression_N12_t0", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 2, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "TXN_vol_20d", "Core_PCE_zscore_60d", "GD_GeneralDynamics_zscore_60d", "AMGN_Amgen_ret_1d", "EWA_Australia_zscore_60d", "EWL_Switzerland_vol_20d", "HD_ret_20d", "AMZN_ret_5d", "JNJ_ret_1d", "ASX_Australia_ret_5d"], "is_new": true}, {"model_id": "new_h2_GLOBAL_LogisticRegression_N12_t1", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 2, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "MSTR_Bitcoin3_ret_1d", "EWL_Switzerland_zscore_60d", "JNJ_ret_1d", "heston_var_ev_h5", "SPY_zscore_60d", "spx_momentum_3d", "Nikkei_Japan_zscore_60d", "VRP_ma5", "ASX_Australia_vol_20d", "heston_ev_h3"], "is_new": true}, {"model_id": "new_h2_GLOBAL_LogisticRegression_N12_t2", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 2, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "CPB_CampbellSoup_ret_20d", "NFCI_ret_5d", "NOC_Northrop_ret_20d", "US3M_Rate_zscore_60d", "HangSeng_HK_ret_5d", "NVDA_vol_20d", "AMT_AmericanTower_ret_1d", "DHR_ret_1d", "heston_ev_h3", "Brent_Oil_FRED_ret_5d"], "is_new": true}, {"model_id": "new_h2_GLOBAL_LogisticRegression_N12_t3", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 2, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "NWL_Newell_ret_20d", "EXC_Exelon_zscore_60d", "DHR_vol_20d", "SLB_Schlumberger_ret_1d", "Core_CPI_zscore_60d", "DOW_Price_zscore_60d", "LLY_zscore_60d", "US3M_Rate_vol_20d", "TED_Spread_vol_20d", "EWA_Australia_ret_1d"], "is_new": true}, {"model_id": "new_h2_GLOBAL_LogisticRegression_N12_t4", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 2, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "GD_GeneralDynamics_zscore_60d", "XOM_ret_1d", "SLB_Schlumberger_ret_5d", "US30Y_Rate_ret_20d", "FedFunds_zscore_60d", "US7Y_Rate_ret_20d", "ITT_ITTInc_ret_5d", "WTI_Oil_FRED_zscore_60d", "Brent_Oil_FRED_ret_5d", "EOG_EOGResources_vol_20d"], "is_new": true}, {"model_id": "new_h2_GLOBAL_LogisticRegression_N12_t5", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 2, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWS_Singapore_ret_5d", "EWM_Malaysia_ret_1d", "CCI_CrownCastle_vol_20d", "EXC_Exelon_zscore_60d", "Nikkei_Japan_zscore_60d", "MO_AltriaMG_ret_1d", "NOC_Northrop_ret_20d", "LMT_LockheedMartin_vol_20d", "SCHW_Schwab_ret_5d", "Core_PCE_zscore_60d"], "is_new": true}, {"model_id": "new_h2_GLOBAL_LogisticRegression_N12_t6", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 2, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "SO_SouthernCo_ret_5d", "gjr_condvar_h1", "CPB_CampbellSoup_ret_20d", "EWJ_Japan_vol_20d", "AMT_AmericanTower_ret_1d", "HD_ret_5d", "US5Y_Rate_ret_5d", "XOM_ret_1d", "LLY_zscore_60d", "BDX_Becton_Dickinson_ret_20d"], "is_new": true}, {"model_id": "new_h2_GLOBAL_LogisticRegression_N12_t7", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 2, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "hmm_p_stress", "CPB_CampbellSoup_ret_5d", "MS_MorganStanley_zscore_60d", "GE_ret_1d", "LMT_LockheedMartin_vol_20d", "AVB_AvalonBay_zscore_60d", "TED_Spread_zscore_60d", "MSTR_Bitcoin3_ret_5d", "DHR_vol_20d", "US3M_Rate_vol_20d"], "is_new": true}, {"model_id": "new_h2_GLOBAL_LogisticRegression_N15_t0", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 2, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWM_Malaysia_ret_1d", "AMT_AmericanTower_ret_1d", "PPL_PPL_ret_1d", "DIS_vol_20d", "GE_ret_1d", "US1Y_Rate_ret_20d", "SBUX_zscore_60d", "MSTR_Bitcoin3_ret_5d", "EXC_Exelon_ret_1d", "CPB_CampbellSoup_ret_20d", "ES_Evergy_ret_1d", "NFCI_ret_5d", "CPB_CampbellSoup_ret_5d"], "is_new": true}, {"model_id": "new_h2_GLOBAL_LogisticRegression_N15_t1", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 2, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWY_Korea_zscore_60d", "ITT_ITTInc_ret_5d", "TXN_vol_20d", "MS_MorganStanley_zscore_60d", "Core_PCE_zscore_60d", "SLB_Schlumberger_ret_1d", "AMD_ret_5d", "EWG_Germany_ret_20d", "US5Y_Rate_ret_5d", "EWS_Singapore_ret_5d", "IBEX_Spain_ret_20d", "ASX_Australia_vol_20d", "SPY_zscore_60d"], "is_new": true}, {"model_id": "new_h2_GLOBAL_LogisticRegression_N15_t2", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 2, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "DHR_vol_20d", "heston_var_ev_h3", "QQQ_vol_20d", "SO_SouthernCo_ret_5d", "TED_Spread_zscore_60d", "LMT_LockheedMartin_vol_20d", "US3M_Rate_vol_20d", "TXN_vol_20d", "SLB_Schlumberger_ret_5d", "DAX_Germany_vol_20d", "VOD_Vodafone_zscore_60d", "EOG_EOGResources_vol_20d", "AMD_ret_5d"], "is_new": true}, {"model_id": "new_h2_GLOBAL_LogisticRegression_N15_t3", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 2, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "XLK_Tech_zscore_60d", "EWC_Canada_zscore_60d", "EQIX_Equinix_ret_5d", "MO_AltriaMG_ret_1d", "vix_acceleration_1d", "WTI_Oil_FRED_zscore_60d", "NOC_Northrop_ret_20d", "DHR_vol_20d", "PFE_ret_1d", "ORCL_vol_20d", "EMR_Emerson_ret_20d", "DHR_ret_1d", "EWQ_France_zscore_60d"], "is_new": true}, {"model_id": "new_h2_GLOBAL_LogisticRegression_N15_t4", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 2, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "PPL_PPL_ret_1d", "JNJ_ret_1d", "EWM_Malaysia_zscore_60d", "EXC_Exelon_ret_1d", "DAX_Germany_vol_20d", "TGT_Target_zscore_60d", "EWG_Germany_vol_20d", "XLK_Tech_zscore_60d", "XLF_Fin_vol_20d", "LLY_zscore_60d", "Industrial_Production_zscore_60d", "heston_ev_h3", "US30Y_Rate_ret_20d"], "is_new": true}, {"model_id": "new_h2_GLOBAL_LogisticRegression_N15_t5", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 2, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "SO_SouthernCo_ret_5d", "AMGN_Amgen_ret_1d", "CLX_Clorox_vol_20d", "PCAR_PaccarInc_ret_5d", "SJM_JM_Smucker_ret_1d", "CCI_CrownCastle_vol_20d", "AMZN_ret_5d", "NVDA_vol_20d", "DE_Deere_ret_5d", "US3M_Rate_vol_20d", "HUM_Humana_ret_5d", "LOW_Lowes_ret_5d", "DE_Deere_vol_20d"], "is_new": true}, {"model_id": "new_h2_GLOBAL_LogisticRegression_N15_t6", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 2, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "PCAR_PaccarInc_ret_5d", "EWM_Malaysia_ret_1d", "LUV_SouthwestAir_ret_5d", "vix_acceleration_1d", "EWJ_Japan_vol_20d", "HD_ret_1d", "CPB_CampbellSoup_ret_20d", "heston_var_ev_h5", "HD_ret_20d", "EWY_Korea_ret_20d", "XOM_ret_1d", "Brent_Oil_FRED_ret_20d", "AORD_AUS_zscore_60d"], "is_new": true}, {"model_id": "new_h2_GLOBAL_LogisticRegression_N15_t7", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 2, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "DAX_Germany_vol_20d", "TED_Spread_zscore_60d", "Nikkei_Japan_zscore_60d", "spx_vol_5d", "Nikkei_Japan_vol_20d", "PLD_Prologis_ret_5d", "US3Y_Rate_ret_5d", "heston_var_ev_h3", "AXP_Amex_vol_20d", "NWL_Newell_ret_20d", "SBUX_vol_20d", "EXC_Exelon_ret_1d", "PPL_PPL_ret_1d"], "is_new": true}, {"model_id": "new_h2_GLOBAL_LogisticRegression_N20_t0", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 2, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "CCI_CrownCastle_vol_20d", "spx_vol_5d", "BLK_BlackRock_zscore_60d", "HUM_Humana_ret_5d", "3M_ret_5d", "US1Y_Rate_ret_20d", "EWM_Malaysia_vol_20d", "SLB_Schlumberger_ret_5d", "EWY_Korea_zscore_60d", "EWY_Korea_ret_20d", "PLD_Prologis_ret_5d", "US1Y_Rate_ret_5d", "EOG_EOGResources_vol_20d", "MS_MorganStanley_ret_5d", "AMGN_Amgen_ret_1d", "CI_Cigna_vol_20d", "heston_var_ev_h5", "TXN_vol_20d"], "is_new": true}, {"model_id": "new_h2_GLOBAL_LogisticRegression_N20_t1", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 2, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "heston_var_ev_h5", "EWL_Switzerland_zscore_60d", "EWA_Australia_ret_1d", "HD_ret_20d", "DHR_vol_20d", "DE_Deere_vol_20d", "IWM_SmallCap_vol_20d", "EWJ_Japan_vol_20d", "VRP_ma5", "EWM_Malaysia_ret_1d", "XLV_Health_zscore_60d", "DE_Deere_ret_5d", "FedFunds_zscore_60d", "DAX_Germany_zscore_60d", "LOW_Lowes_ret_5d", "SCHW_Schwab_ret_5d", "Retail_Sales_zscore_60d", "PAYX_Paychex_zscore_60d"], "is_new": true}, {"model_id": "new_h2_GLOBAL_LogisticRegression_N20_t2", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 2, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "LOW_Lowes_ret_5d", "PAYX_Paychex_vol_20d", "Industrial_Production_zscore_60d", "AXP_Amex_vol_20d", "ENB_EnbridgeInc_ret_1d", "CPB_CampbellSoup_zscore_60d", "spx_momentum_3d", "T10Y2Y_Spread_ret_5d", "TED_Spread_vol_20d", "LLY_zscore_60d", "ES_Evergy_ret_1d", "CMCSA_ret_1d", "CPB_CampbellSoup_ret_5d", "HD_ret_20d", "IBEX_Spain_ret_20d", "Nikkei_Japan_zscore_60d", "ASX_Australia_ret_5d", "EWM_Malaysia_vol_20d"], "is_new": true}, {"model_id": "new_h2_GLOBAL_LogisticRegression_N20_t3", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 2, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "US7Y_Rate_ret_20d", "MS_MorganStanley_zscore_60d", "TGT_Target_zscore_60d", "SBUX_zscore_60d", "TM_Telephone_vol_20d", "SLB_Schlumberger_ret_1d", "HangSeng_HK_ret_1d", "EQIX_Equinix_ret_5d", "AMGN_Amgen_ret_1d", "vix_acceleration_1d", "HD_ret_5d", "PG_ret_20d", "IYM_BasicMaterials_ret_20d", "XOM_ret_20d", "VOD_Vodafone_zscore_60d", "US30Y_Rate_ret_20d", "EWY_Korea_ret_20d", "MO_AltriaMG_ret_1d"], "is_new": true}, {"model_id": "new_h2_GLOBAL_LogisticRegression_N20_t4", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 2, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "BDX_Becton_Dickinson_ret_20d", "MSTR_Bitcoin3_ret_5d", "US5Y_Rate_ret_5d", "EWS_Singapore_ret_5d", "VOD_Vodafone_zscore_60d", "EQIX_Equinix_ret_5d", "TED_Spread_vol_20d", "JNJ_ret_1d", "EWQ_France_zscore_60d", "DE_Deere_ret_5d", "heston_var_ev_h7", "heston_var_ev_h3", "BA_ret_1d", "DOW_Price_zscore_60d", "INTC_ret_5d", "IBEX_Spain_ret_20d", "NEE_NextEra_ret_20d", "CCI_CrownCastle_vol_20d"], "is_new": true}, {"model_id": "new_h2_GLOBAL_LogisticRegression_N20_t5", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 2, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "PAYX_Paychex_ret_20d", "heston_var_ev_h7", "PAYX_Paychex_zscore_60d", "TED_Spread_vol_20d", "vix_acceleration_1d", "BTI_BritishAmerican_ret_5d", "T10Y2Y_Spread_ret_5d", "US3M_Rate_zscore_60d", "SPY_zscore_60d", "ORCL_vol_20d", "EWL_Switzerland_vol_20d", "NOC_Northrop_ret_20d", "CCI_CrownCastle_vol_20d", "EWM_Malaysia_ret_1d", "US3Y_Rate_ret_5d", "IWM_SmallCap_vol_20d", "XLY_Disc_vol_20d", "HD_zscore_60d"], "is_new": true}, {"model_id": "new_h2_GLOBAL_LogisticRegression_N20_t6", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 2, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "Core_CPI_zscore_60d", "US30Y_Rate_ret_20d", "IYR_US_REIT2_zscore_60d", "ORCL_zscore_60d", "EOG_EOGResources_vol_20d", "EWM_Malaysia_vol_20d", "gjr_condvar_h1", "BA_ret_1d", "CLX_Clorox_vol_20d", "XLY_Disc_vol_20d", "HD_ret_1d", "PFE_ret_1d", "EWJ_Japan_vol_20d", "SCHW_Schwab_ret_5d", "SBUX_vol_20d", "US3Y_Rate_ret_5d", "NWL_Newell_ret_20d", "DHR_ret_1d"], "is_new": true}, {"model_id": "new_h2_GLOBAL_LogisticRegression_N20_t7", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 2, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "HD_zscore_60d", "US3M_Rate_vol_20d", "XOM_ret_1d", "SCHW_Schwab_ret_5d", "PG_ret_20d", "US3M_Rate_zscore_60d", "CI_Cigna_vol_20d", "Retail_Sales_zscore_60d", "US1Y_Rate_ret_20d", "XOM_ret_20d", "gjr_condvar_h1", "EFFR_ret_1d", "DE_Deere_vol_20d", "AMT_AmericanTower_ret_1d", "AXP_Amex_ret_20d", "T_ret_1d", "heston_var_ev_h7", "DAX_Germany_vol_20d"], "is_new": true}, {"model_id": "new_h2_GLOBAL_LogisticRegression_N25_t0", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 2, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EQIX_Equinix_ret_5d", "SLB_Schlumberger_ret_1d", "NEE_NextEra_ret_20d", "NOC_Northrop_ret_20d", "VVIX_ret_20d", "BDX_Becton_Dickinson_ret_20d", "MS_MorganStanley_zscore_60d", "EQR_Equity_ret_1d", "INTC_ret_5d", "CI_Cigna_vol_20d", "ITT_ITTInc_ret_5d", "SBUX_zscore_60d", "gjr_condvar_h1", "VOD_Vodafone_zscore_60d", "JNJ_ret_1d", "CPB_CampbellSoup_zscore_60d", "EWY_Korea_zscore_60d", "US7Y_Rate_ret_20d", "AVB_AvalonBay_zscore_60d", "US3M_Rate_vol_20d", "heston_var_ev_h7", "PAYX_Paychex_ret_20d", "CPB_CampbellSoup_ret_5d"], "is_new": true}, {"model_id": "new_h2_GLOBAL_LogisticRegression_N25_t1", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 2, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "FedFunds_zscore_60d", "gjr_condvar_h1", "LMT_LockheedMartin_vol_20d", "SO_SouthernCo_ret_5d", "SBUX_ret_5d", "EQR_Equity_ret_1d", "GE_ret_1d", "EFFR_vol_20d", "EOG_EOGResources_vol_20d", "spx_abs_ret_max_5d", "HUM_Humana_ret_5d", "VOD_Vodafone_zscore_60d", "MS_MorganStanley_ret_5d", "NWL_Newell_ret_20d", "LMT_LockheedMartin_ret_1d", "DAX_Germany_zscore_60d", "HangSeng_HK_ret_1d", "SPY_zscore_60d", "EWC_Canada_zscore_60d", "AXP_Amex_ret_20d", "LLY_zscore_60d", "Core_CPI_zscore_60d", "PCAR_PaccarInc_ret_5d"], "is_new": true}, {"model_id": "new_h2_GLOBAL_LogisticRegression_N25_t2", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 2, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "BTI_BritishAmerican_ret_20d", "EQR_Equity_ret_1d", "spx_abs_ret_max_5d", "EWL_Switzerland_vol_20d", "EXC_Exelon_ret_1d", "EWG_Germany_vol_20d", "EWQ_France_ret_20d", "IWM_SmallCap_vol_20d", "EWM_Malaysia_zscore_60d", "Nikkei_Japan_zscore_60d", "PG_ret_20d", "MRK_Merck_zscore_60d", "PAYX_Paychex_vol_20d", "SBUX_ret_5d", "SJM_JM_Smucker_ret_1d", "PAYX_Paychex_ret_20d", "EWJ_Japan_vol_20d", "EWY_Korea_zscore_60d", "DE_Deere_ret_5d", "MS_MorganStanley_ret_1d", "IYR_US_REIT2_zscore_60d", "TGT_Target_zscore_60d", "XLY_Disc_vol_20d"], "is_new": true}, {"model_id": "new_h2_GLOBAL_LogisticRegression_N25_t3", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 2, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "AMD_ret_1d", "HangSeng_HK_ret_1d", "heston_var_ev_h7", "VRP_ma5", "ORCL_zscore_60d", "WTI_Oil_FRED_zscore_60d", "CMCSA_ret_1d", "EXC_Exelon_ret_1d", "3M_ret_5d", "PPL_PPL_ret_1d", "Retail_Sales_zscore_60d", "Brent_Oil_FRED_ret_5d", "EQR_Equity_ret_1d", "SLB_Schlumberger_ret_5d", "AORD_AUS_zscore_60d", "NFCI_ret_5d", "CI_Cigna_vol_20d", "US5Y_Rate_ret_5d", "VVIX_ret_20d", "ASX_Australia_vol_20d", "SBUX_ret_5d", "FedFunds_zscore_60d", "MS_MorganStanley_ret_5d"], "is_new": true}, {"model_id": "new_h2_GLOBAL_LogisticRegression_N25_t4", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 2, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "QQQ_vol_20d", "DOW_Price_zscore_60d", "CMCSA_ret_1d", "PFE_ret_1d", "Core_PCE_zscore_60d", "EOG_EOGResources_ret_5d", "HD_zscore_60d", "EWM_Malaysia_vol_20d", "MO_AltriaMG_ret_1d", "MS_MorganStanley_ret_1d", "NWL_Newell_ret_20d", "SJM_JM_Smucker_ret_5d", "EWS_Singapore_ret_5d", "AXP_Amex_vol_20d", "EWL_Switzerland_vol_20d", "EFFR_ret_1d", "LMT_LockheedMartin_ret_1d", "EWC_Canada_zscore_60d", "heston_ev_h3", "US3Y_Rate_ret_5d", "MRK_Merck_zscore_60d", "DHR_vol_20d", "LOW_Lowes_ret_5d"], "is_new": true}, {"model_id": "new_h2_GLOBAL_LogisticRegression_N25_t5", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 2, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWG_Germany_vol_20d", "INTC_ret_5d", "AMD_ret_1d", "QQQ_vol_20d", "US30Y_Rate_ret_20d", "CMCSA_ret_1d", "BDX_Becton_Dickinson_ret_20d", "SJM_JM_Smucker_ret_1d", "EQIX_Equinix_ret_5d", "T10Y2Y_Spread_ret_5d", "EWM_Malaysia_ret_1d", "VVIX_ret_20d", "HangSeng_HK_ret_1d", "PG_ret_20d", "MSTR_Bitcoin3_ret_5d", "GD_GeneralDynamics_zscore_60d", "NOC_Northrop_ret_20d", "Brent_Oil_FRED_ret_20d", "EQR_Equity_ret_1d", "MSTR_Bitcoin3_ret_1d", "CLX_Clorox_vol_20d", "EXC_Exelon_zscore_60d", "vix_mean_abs_ret_5d"], "is_new": true}, {"model_id": "new_h2_GLOBAL_LogisticRegression_N25_t6", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 2, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "US3Y_Rate_ret_5d", "MS_MorganStanley_zscore_60d", "EQIX_Equinix_ret_5d", "ORCL_zscore_60d", "IYR_US_REIT2_zscore_60d", "MS_MorganStanley_ret_5d", "HangSeng_HK_ret_1d", "EXC_Exelon_ret_1d", "EWM_Malaysia_ret_1d", "US6M_Rate_ret_20d", "ASX_Australia_vol_20d", "US7Y_Rate_ret_20d", "INTC_ret_5d", "BTI_BritishAmerican_ret_20d", "XLB_Materials_zscore_60d", "Industrial_Production_zscore_60d", "hmm_p_stress", "M_Macys_vol_20d", "EWL_Switzerland_vol_20d", "AMGN_Amgen_ret_1d", "BLK_BlackRock_zscore_60d", "XLF_Fin_vol_20d", "EWG_Germany_ret_20d"], "is_new": true}, {"model_id": "new_h2_GLOBAL_LogisticRegression_N25_t7", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 2, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "LOW_Lowes_ret_20d", "Brent_Oil_FRED_ret_5d", "MS_MorganStanley_ret_1d", "EXC_Exelon_zscore_60d", "BDX_Becton_Dickinson_ret_20d", "PAYX_Paychex_vol_20d", "SLB_Schlumberger_ret_5d", "PG_ret_20d", "spx_abs_ret_max_5d", "HangSeng_HK_ret_1d", "SJM_JM_Smucker_ret_5d", "CPB_CampbellSoup_zscore_60d", "CMCSA_ret_1d", "EWG_Germany_vol_20d", "LUV_SouthwestAir_ret_5d", "EWC_Canada_zscore_60d", "QQQ_vol_20d", "EWY_Korea_ret_20d", "NOC_Northrop_ret_20d", "XLV_Health_zscore_60d", "EFFR_vol_20d", "SBUX_ret_5d", "EMR_Emerson_ret_20d"], "is_new": true}, {"model_id": "new_h2_GLOBAL_LogisticRegression_N30_t0", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 2, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "PFE_ret_1d", "PPL_PPL_ret_1d", "BLK_BlackRock_zscore_60d", "IYR_US_REIT2_zscore_60d", "LOW_Lowes_ret_5d", "HangSeng_HK_ret_1d", "SPY_zscore_60d", "VRP_ma5", "US7Y_Rate_ret_20d", "ASX_Australia_vol_20d", "QQQ_vol_20d", "CLX_Clorox_vol_20d", "EQR_Equity_ret_1d", "VOD_Vodafone_zscore_60d", "XLB_Materials_zscore_60d", "MSTR_Bitcoin3_ret_5d", "XLF_Fin_vol_20d", "WTI_Oil_FRED_zscore_60d", "CPB_CampbellSoup_ret_20d", "IWM_SmallCap_vol_20d", "HD_zscore_60d", "T_ret_1d", "XOM_ret_1d", "Retail_Sales_zscore_60d", "AMD_ret_1d", "AXP_Amex_vol_20d", "DAX_Germany_zscore_60d", "EWY_Korea_ret_20d"], "is_new": true}, {"model_id": "new_h2_GLOBAL_LogisticRegression_N30_t1", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 2, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "MS_MorganStanley_ret_5d", "SLB_Schlumberger_ret_5d", "US3Y_Rate_ret_5d", "EWA_Australia_zscore_60d", "PPL_PPL_ret_1d", "FedFunds_zscore_60d", "NVDA_vol_20d", "AMT_AmericanTower_ret_1d", "EFFR_vol_20d", "BTI_BritishAmerican_ret_20d", "EWY_Korea_ret_20d", "ORCL_vol_20d", "TXN_vol_20d", "TM_Telephone_ret_1d", "HD_ret_5d", "Brent_Oil_FRED_ret_5d", "LOW_Lowes_ret_5d", "SJM_JM_Smucker_ret_1d", "SBUX_ret_5d", "CI_Cigna_vol_20d", "DE_Deere_vol_20d", "BTI_BritishAmerican_ret_5d", "TED_Spread_vol_20d", "GD_GeneralDynamics_zscore_60d", "EOG_EOGResources_vol_20d", "MRK_Merck_zscore_60d", "heston_ev_h3", "LMT_LockheedMartin_vol_20d"], "is_new": true}, {"model_id": "new_h2_GLOBAL_LogisticRegression_N30_t2", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 2, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "vix_acceleration_1d", "LUV_SouthwestAir_ret_5d", "Industrial_Production_zscore_60d", "US1Y_Rate_ret_5d", "WTI_Oil_FRED_zscore_60d", "EFFR_vol_20d", "EFFR_ret_1d", "EWL_Switzerland_zscore_60d", "MO_AltriaMG_ret_1d", "LOW_Lowes_ret_5d", "DE_Deere_ret_5d", "TXN_vol_20d", "ITT_ITTInc_ret_5d", "spx_momentum_3d", "SBUX_zscore_60d", "AMZN_ret_5d", "ORCL_vol_20d", "EWA_Australia_zscore_60d", "3M_vol_20d", "DOW_Price_zscore_60d", "DIS_vol_20d", "T_ret_1d", "EQIX_Equinix_ret_5d", "EWM_Malaysia_ret_1d", "EWQ_France_ret_20d", "LOW_Lowes_ret_20d", "DHR_vol_20d", "XOM_ret_20d"], "is_new": true}, {"model_id": "new_h2_GLOBAL_LogisticRegression_N30_t3", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 2, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWA_Australia_zscore_60d", "LMT_LockheedMartin_ret_1d", "WTI_Oil_FRED_zscore_60d", "hmm_p_stress", "NEE_NextEra_ret_20d", "HangSeng_HK_ret_1d", "SBUX_vol_20d", "EFFR_vol_20d", "NFCI_ret_5d", "TED_Spread_vol_20d", "IBEX_Spain_ret_20d", "SJM_JM_Smucker_ret_1d", "GD_GeneralDynamics_zscore_60d", "heston_var_ev_h3", "GILD_Gilead_ret_20d", "LOW_Lowes_ret_5d", "ITT_ITTInc_ret_5d", "ASX_Australia_ret_5d", "PCAR_PaccarInc_ret_5d", "DOW_Price_zscore_60d", "T10Y2Y_Spread_ret_5d", "EWS_Singapore_ret_5d", "CPB_CampbellSoup_vol_20d", "AMD_ret_1d", "DAX_Germany_vol_20d", "T_ret_1d", "LMT_LockheedMartin_vol_20d", "AXP_Amex_vol_20d"], "is_new": true}, {"model_id": "new_h2_GLOBAL_LogisticRegression_N30_t4", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 2, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "Michigan_Sentiment_ret_20d", "IBEX_Spain_ret_20d", "LMT_LockheedMartin_ret_1d", "AMZN_ret_5d", "Industrial_Production_zscore_60d", "TM_Telephone_vol_20d", "CTAS_Cintas_vol_20d", "AMD_ret_5d", "INTC_ret_5d", "TED_Spread_zscore_60d", "AMD_ret_1d", "CPB_CampbellSoup_zscore_60d", "3M_ret_5d", "US30Y_Rate_ret_20d", "US1Y_Rate_ret_5d", "NVDA_vol_20d", "SBUX_vol_20d", "VVIX_ret_20d", "SBUX_zscore_60d", "EXC_Exelon_zscore_60d", "EWY_Korea_zscore_60d", "SJM_JM_Smucker_ret_1d", "ORCL_vol_20d", "CCI_CrownCastle_vol_20d", "EWJ_Japan_vol_20d", "PAYX_Paychex_vol_20d", "TGT_Target_zscore_60d", "T_ret_1d"], "is_new": true}, {"model_id": "new_h2_GLOBAL_LogisticRegression_N30_t5", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 2, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "heston_ev_h3", "HUM_Humana_ret_5d", "MS_MorganStanley_ret_1d", "IYM_BasicMaterials_ret_20d", "EWS_Singapore_ret_5d", "EWL_Switzerland_zscore_60d", "EWM_Malaysia_zscore_60d", "HangSeng_HK_ret_1d", "PAYX_Paychex_ret_20d", "heston_var_ev_h3", "GILD_Gilead_ret_20d", "US3Y_Rate_ret_5d", "NEE_NextEra_ret_20d", "EWM_Malaysia_vol_20d", "EWC_Canada_zscore_60d", "Retail_Sales_zscore_60d", "US5Y_Rate_ret_5d", "DIS_vol_20d", "EFFR_ret_1d", "Nikkei_Japan_zscore_60d", "EOG_EOGResources_ret_5d", "SJM_JM_Smucker_ret_5d", "Core_CPI_zscore_60d", "Brent_Oil_FRED_ret_5d", "LLY_zscore_60d", "3M_ret_5d", "EQR_Equity_ret_1d", "EWL_Switzerland_vol_20d"], "is_new": true}, {"model_id": "new_h2_GLOBAL_LogisticRegression_N30_t6", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 2, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWS_Singapore_ret_5d", "HD_ret_5d", "INTC_ret_1d", "AXP_Amex_vol_20d", "ITT_ITTInc_ret_5d", "ENB_EnbridgeInc_ret_1d", "Core_CPI_zscore_60d", "WTI_Oil_FRED_zscore_60d", "HD_zscore_60d", "AMD_ret_1d", "INTC_ret_5d", "3M_ret_5d", "EWY_Korea_zscore_60d", "QQQ_vol_20d", "DE_Deere_ret_5d", "NOC_Northrop_ret_20d", "AMZN_ret_5d", "Retail_Sales_zscore_60d", "US3M_Rate_vol_20d", "Nikkei_Japan_vol_20d", "EMR_Emerson_ret_20d", "Brent_Oil_FRED_ret_5d", "GE_ret_1d", "LMT_LockheedMartin_vol_20d", "SBUX_vol_20d", "PAYX_Paychex_zscore_60d", "EWQ_France_ret_20d", "NEE_NextEra_ret_20d"], "is_new": true}, {"model_id": "new_h2_GLOBAL_LogisticRegression_N30_t7", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 2, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "XLF_Fin_vol_20d", "US1Y_Rate_ret_5d", "CMCSA_ret_1d", "ORCL_zscore_60d", "HangSeng_HK_ret_1d", "SBUX_vol_20d", "EWM_Malaysia_zscore_60d", "MS_MorganStanley_ret_1d", "ORCL_vol_20d", "LUV_SouthwestAir_ret_5d", "EWG_Germany_vol_20d", "XLV_Health_zscore_60d", "spx_momentum_3d", "TED_Spread_zscore_60d", "TED_Spread_vol_20d", "LMT_LockheedMartin_ret_1d", "XOM_ret_20d", "PAYX_Paychex_zscore_60d", "spx_abs_ret_max_5d", "SLB_Schlumberger_ret_1d", "LLY_zscore_60d", "LOW_Lowes_ret_5d", "XLB_Materials_zscore_60d", "Core_CPI_zscore_60d", "EWS_Singapore_ret_5d", "CTAS_Cintas_vol_20d", "TM_Telephone_ret_1d", "EWY_Korea_zscore_60d"], "is_new": true}, {"model_id": "new_h3_CALM_XGBoost_N5_t0", "algo": "XGBoost", "regime": "CALM", "horizon": 3, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "ORCL_vol_20d", "EWS_Singapore_ret_5d", "US3M_Rate_vol_20d"], "is_new": true}, {"model_id": "new_h3_CALM_XGBoost_N5_t1", "algo": "XGBoost", "regime": "CALM", "horizon": 3, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWY_Korea_zscore_60d", "Retail_Sales_zscore_60d", "SBUX_ret_5d"], "is_new": true}, {"model_id": "new_h3_CALM_XGBoost_N5_t2", "algo": "XGBoost", "regime": "CALM", "horizon": 3, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "CCI_CrownCastle_vol_20d", "DHR_vol_20d", "EWL_Switzerland_zscore_60d"], "is_new": true}, {"model_id": "new_h3_CALM_XGBoost_N5_t3", "algo": "XGBoost", "regime": "CALM", "horizon": 3, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "MO_AltriaMG_ret_1d", "MSTR_Bitcoin3_ret_5d", "Brent_Oil_FRED_ret_20d"], "is_new": true}, {"model_id": "new_h3_CALM_XGBoost_N5_t4", "algo": "XGBoost", "regime": "CALM", "horizon": 3, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "MRK_Merck_zscore_60d", "SCHW_Schwab_ret_5d", "VRP_ma5"], "is_new": true}, {"model_id": "new_h3_CALM_XGBoost_N5_t5", "algo": "XGBoost", "regime": "CALM", "horizon": 3, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "VVIX_ret_20d", "AVB_AvalonBay_zscore_60d", "heston_var_ev_h3"], "is_new": true}, {"model_id": "new_h3_CALM_XGBoost_N5_t6", "algo": "XGBoost", "regime": "CALM", "horizon": 3, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "INTC_ret_5d", "HD_ret_5d", "LMT_LockheedMartin_ret_1d"], "is_new": true}, {"model_id": "new_h3_CALM_XGBoost_N5_t7", "algo": "XGBoost", "regime": "CALM", "horizon": 3, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "MSTR_Bitcoin3_ret_20d", "ITT_ITTInc_ret_5d", "T_ret_1d"], "is_new": true}, {"model_id": "new_h3_CALM_XGBoost_N8_t0", "algo": "XGBoost", "regime": "CALM", "horizon": 3, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "US3M_Rate_zscore_60d", "BTI_BritishAmerican_ret_5d", "US3Y_Rate_ret_5d", "EWM_Malaysia_ret_1d", "GILD_Gilead_ret_20d", "HangSeng_HK_ret_5d"], "is_new": true}, {"model_id": "new_h3_CALM_XGBoost_N8_t1", "algo": "XGBoost", "regime": "CALM", "horizon": 3, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "spx_momentum_3d", "PPL_PPL_ret_1d", "CPB_CampbellSoup_vol_20d", "DE_Deere_vol_20d", "EWA_Australia_ret_1d", "XLK_Tech_zscore_60d"], "is_new": true}, {"model_id": "new_h3_CALM_XGBoost_N8_t2", "algo": "XGBoost", "regime": "CALM", "horizon": 3, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "Industrial_Production_zscore_60d", "EWQ_France_ret_20d", "EWG_Germany_ret_20d", "MRK_Merck_zscore_60d", "EWA_Australia_zscore_60d", "MO_AltriaMG_ret_1d"], "is_new": true}, {"model_id": "new_h3_CALM_XGBoost_N8_t3", "algo": "XGBoost", "regime": "CALM", "horizon": 3, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "US3Y_Rate_ret_5d", "MS_MorganStanley_zscore_60d", "EFFR_vol_20d", "SJM_JM_Smucker_ret_5d", "ES_Evergy_ret_1d", "Core_PCE_zscore_60d"], "is_new": true}, {"model_id": "new_h3_CALM_XGBoost_N8_t4", "algo": "XGBoost", "regime": "CALM", "horizon": 3, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "US30Y_Rate_ret_20d", "SBUX_zscore_60d", "gjr_condvar_h1", "IWM_SmallCap_vol_20d", "heston_ev_h3", "PPL_PPL_ret_1d"], "is_new": true}, {"model_id": "new_h3_CALM_XGBoost_N8_t5", "algo": "XGBoost", "regime": "CALM", "horizon": 3, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "PLD_Prologis_ret_5d", "NWL_Newell_ret_20d", "EWY_Korea_ret_20d", "AXP_Amex_vol_20d", "VOD_Vodafone_zscore_60d", "PPL_PPL_ret_1d"], "is_new": true}, {"model_id": "new_h3_CALM_XGBoost_N8_t6", "algo": "XGBoost", "regime": "CALM", "horizon": 3, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "MSTR_Bitcoin3_ret_5d", "NFCI_ret_5d", "AMZN_ret_5d", "US3M_Rate_zscore_60d", "SJM_JM_Smucker_ret_1d", "spx_momentum_3d"], "is_new": true}, {"model_id": "new_h3_CALM_XGBoost_N8_t7", "algo": "XGBoost", "regime": "CALM", "horizon": 3, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "AMD_ret_1d", "BDX_Becton_Dickinson_ret_20d", "heston_var_ev_h5", "HD_zscore_60d", "HD_ret_1d", "LUV_SouthwestAir_ret_5d"], "is_new": true}, {"model_id": "new_h3_CALM_XGBoost_N10_t0", "algo": "XGBoost", "regime": "CALM", "horizon": 3, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "WTI_Oil_FRED_zscore_60d", "M_Macys_vol_20d", "EFFR_vol_20d", "XLK_Tech_zscore_60d", "AMZN_ret_5d", "CCI_CrownCastle_vol_20d", "AMGN_Amgen_ret_1d", "PAYX_Paychex_ret_20d"], "is_new": true}, {"model_id": "new_h3_CALM_XGBoost_N10_t1", "algo": "XGBoost", "regime": "CALM", "horizon": 3, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "Michigan_Sentiment_ret_20d", "LOW_Lowes_ret_20d", "DIS_vol_20d", "PAYX_Paychex_vol_20d", "SPY_zscore_60d", "EFFR_vol_20d", "LLY_zscore_60d", "HD_zscore_60d"], "is_new": true}, {"model_id": "new_h3_CALM_XGBoost_N10_t2", "algo": "XGBoost", "regime": "CALM", "horizon": 3, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "XOM_ret_1d", "HangSeng_HK_ret_5d", "vix_mean_abs_ret_5d", "CCI_CrownCastle_vol_20d", "LMT_LockheedMartin_ret_1d", "NWL_Newell_ret_20d", "SJM_JM_Smucker_ret_1d", "Core_PCE_zscore_60d"], "is_new": true}, {"model_id": "new_h3_CALM_XGBoost_N10_t3", "algo": "XGBoost", "regime": "CALM", "horizon": 3, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWG_Germany_vol_20d", "US7Y_Rate_ret_20d", "US1Y_Rate_ret_20d", "PAYX_Paychex_ret_20d", "ENB_EnbridgeInc_ret_1d", "HD_ret_1d", "heston_var_ev_h3", "INTC_ret_1d"], "is_new": true}, {"model_id": "new_h3_CALM_XGBoost_N10_t4", "algo": "XGBoost", "regime": "CALM", "horizon": 3, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "SLB_Schlumberger_ret_1d", "vix_mean_abs_ret_5d", "SJM_JM_Smucker_ret_5d", "HangSeng_HK_ret_5d", "3M_vol_20d", "SCHW_Schwab_ret_5d", "IYM_BasicMaterials_ret_20d", "INTC_ret_1d"], "is_new": true}, {"model_id": "new_h3_CALM_XGBoost_N10_t5", "algo": "XGBoost", "regime": "CALM", "horizon": 3, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "MS_MorganStanley_ret_5d", "Industrial_Production_zscore_60d", "XLY_Disc_vol_20d", "XOM_ret_20d", "Retail_Sales_zscore_60d", "DHR_vol_20d", "ORCL_vol_20d", "T10Y2Y_Spread_ret_5d"], "is_new": true}, {"model_id": "new_h3_CALM_XGBoost_N10_t6", "algo": "XGBoost", "regime": "CALM", "horizon": 3, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "AMZN_ret_5d", "LMT_LockheedMartin_ret_1d", "CI_Cigna_vol_20d", "EWA_Australia_ret_1d", "AXP_Amex_ret_20d", "NEE_NextEra_ret_20d", "XLF_Fin_vol_20d", "NOC_Northrop_ret_20d"], "is_new": true}, {"model_id": "new_h3_CALM_XGBoost_N10_t7", "algo": "XGBoost", "regime": "CALM", "horizon": 3, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "TGT_Target_zscore_60d", "CPB_CampbellSoup_zscore_60d", "INTC_ret_5d", "MS_MorganStanley_ret_1d", "T_ret_1d", "EWS_Singapore_ret_5d", "US1Y_Rate_ret_20d", "PAYX_Paychex_zscore_60d"], "is_new": true}, {"model_id": "new_h3_CALM_XGBoost_N12_t0", "algo": "XGBoost", "regime": "CALM", "horizon": 3, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWM_Malaysia_ret_1d", "PAYX_Paychex_zscore_60d", "US3M_Rate_zscore_60d", "US1Y_Rate_ret_5d", "EFFR_ret_1d", "CTAS_Cintas_vol_20d", "HD_ret_5d", "US3Y_Rate_ret_5d", "EOG_EOGResources_ret_5d", "IBEX_Spain_ret_20d"], "is_new": true}, {"model_id": "new_h3_CALM_XGBoost_N12_t1", "algo": "XGBoost", "regime": "CALM", "horizon": 3, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "Retail_Sales_zscore_60d", "EWQ_France_zscore_60d", "3M_ret_5d", "Nikkei_Japan_vol_20d", "EWA_Australia_zscore_60d", "US3M_Rate_vol_20d", "ASX_Australia_vol_20d", "TED_Spread_zscore_60d", "MRK_Merck_zscore_60d", "EWL_Switzerland_vol_20d"], "is_new": true}, {"model_id": "new_h3_CALM_XGBoost_N12_t2", "algo": "XGBoost", "regime": "CALM", "horizon": 3, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "NWL_Newell_ret_20d", "SPY_zscore_60d", "DAX_Germany_vol_20d", "PAYX_Paychex_ret_20d", "EMR_Emerson_ret_20d", "EWC_Canada_zscore_60d", "TM_Telephone_ret_1d", "MS_MorganStanley_zscore_60d", "Industrial_Production_zscore_60d", "vix_acceleration_1d"], "is_new": true}, {"model_id": "new_h3_CALM_XGBoost_N12_t3", "algo": "XGBoost", "regime": "CALM", "horizon": 3, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "SBUX_ret_5d", "MRK_Merck_zscore_60d", "LMT_LockheedMartin_ret_1d", "US3M_Rate_vol_20d", "LOW_Lowes_ret_20d", "PAYX_Paychex_vol_20d", "EWS_Singapore_ret_5d", "IWM_SmallCap_vol_20d", "HD_zscore_60d", "CPB_CampbellSoup_ret_5d"], "is_new": true}, {"model_id": "new_h3_CALM_XGBoost_N12_t4", "algo": "XGBoost", "regime": "CALM", "horizon": 3, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "spx_momentum_3d", "DE_Deere_ret_5d", "LMT_LockheedMartin_vol_20d", "AMD_ret_5d", "HUM_Humana_ret_5d", "AVB_AvalonBay_zscore_60d", "EQIX_Equinix_ret_5d", "IYM_BasicMaterials_ret_20d", "TGT_Target_zscore_60d", "hmm_p_stress"], "is_new": true}, {"model_id": "new_h3_CALM_XGBoost_N12_t5", "algo": "XGBoost", "regime": "CALM", "horizon": 3, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "IWM_SmallCap_vol_20d", "TED_Spread_zscore_60d", "PAYX_Paychex_vol_20d", "DOW_Price_zscore_60d", "EWJ_Japan_vol_20d", "JNJ_ret_1d", "SBUX_vol_20d", "SJM_JM_Smucker_ret_5d", "HD_zscore_60d", "LOW_Lowes_ret_5d"], "is_new": true}, {"model_id": "new_h3_CALM_XGBoost_N12_t6", "algo": "XGBoost", "regime": "CALM", "horizon": 3, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EFFR_ret_1d", "HD_ret_5d", "Core_CPI_zscore_60d", "AMGN_Amgen_ret_1d", "CI_Cigna_vol_20d", "MO_AltriaMG_ret_1d", "BLK_BlackRock_zscore_60d", "PLD_Prologis_ret_5d", "HangSeng_HK_ret_5d", "US6M_Rate_ret_20d"], "is_new": true}, {"model_id": "new_h3_CALM_XGBoost_N12_t7", "algo": "XGBoost", "regime": "CALM", "horizon": 3, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "AMD_ret_5d", "T10Y2Y_Spread_ret_5d", "vix_mean_abs_ret_5d", "Core_PCE_zscore_60d", "Nikkei_Japan_vol_20d", "Core_CPI_zscore_60d", "CLX_Clorox_vol_20d", "JNJ_ret_1d", "heston_ev_h3", "LUV_SouthwestAir_ret_5d"], "is_new": true}, {"model_id": "new_h3_CALM_XGBoost_N15_t0", "algo": "XGBoost", "regime": "CALM", "horizon": 3, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWM_Malaysia_zscore_60d", "heston_var_ev_h7", "HD_ret_20d", "AMGN_Amgen_ret_1d", "GILD_Gilead_ret_20d", "ORCL_zscore_60d", "spx_vol_5d", "spx_abs_ret_max_5d", "LOW_Lowes_ret_20d", "AXP_Amex_ret_20d", "AVB_AvalonBay_zscore_60d", "IBEX_Spain_ret_20d", "IYR_US_REIT2_zscore_60d"], "is_new": true}, {"model_id": "new_h3_CALM_XGBoost_N15_t1", "algo": "XGBoost", "regime": "CALM", "horizon": 3, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "DE_Deere_ret_5d", "XOM_ret_20d", "Retail_Sales_zscore_60d", "AMT_AmericanTower_ret_1d", "Michigan_Sentiment_ret_20d", "US3Y_Rate_ret_5d", "XOM_ret_1d", "ASX_Australia_vol_20d", "hmm_p_stress", "MSTR_Bitcoin3_ret_5d", "IBEX_Spain_ret_20d", "XLF_Fin_vol_20d", "HangSeng_HK_vol_20d"], "is_new": true}, {"model_id": "new_h3_CALM_XGBoost_N15_t2", "algo": "XGBoost", "regime": "CALM", "horizon": 3, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "hmm_p_stress", "LMT_LockheedMartin_ret_1d", "SLB_Schlumberger_ret_1d", "AVB_AvalonBay_zscore_60d", "EWM_Malaysia_ret_1d", "spx_abs_ret_max_5d", "MS_MorganStanley_ret_5d", "EWJ_Japan_vol_20d", "US30Y_Rate_ret_20d", "EWL_Switzerland_vol_20d", "Michigan_Sentiment_ret_20d", "HD_zscore_60d", "SBUX_vol_20d"], "is_new": true}, {"model_id": "new_h3_CALM_XGBoost_N15_t3", "algo": "XGBoost", "regime": "CALM", "horizon": 3, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "IWM_SmallCap_vol_20d", "SLB_Schlumberger_ret_5d", "CPB_CampbellSoup_ret_20d", "SBUX_vol_20d", "ASX_Australia_vol_20d", "TED_Spread_vol_20d", "EWL_Switzerland_zscore_60d", "EWQ_France_ret_20d", "LMT_LockheedMartin_ret_1d", "BTI_BritishAmerican_ret_20d", "HUM_Humana_ret_5d", "heston_var_ev_h5", "vix_acceleration_1d"], "is_new": true}, {"model_id": "new_h3_CALM_XGBoost_N15_t4", "algo": "XGBoost", "regime": "CALM", "horizon": 3, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "M_Macys_vol_20d", "EWY_Korea_ret_20d", "vix_mean_abs_ret_5d", "XLF_Fin_vol_20d", "ASX_Australia_vol_20d", "AVB_AvalonBay_zscore_60d", "ASX_Australia_ret_5d", "EXC_Exelon_zscore_60d", "LOW_Lowes_ret_20d", "EWJ_Japan_vol_20d", "VRP_ma5", "T_ret_1d", "AMGN_Amgen_ret_1d"], "is_new": true}, {"model_id": "new_h3_CALM_XGBoost_N15_t5", "algo": "XGBoost", "regime": "CALM", "horizon": 3, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "INTC_ret_1d", "CPB_CampbellSoup_vol_20d", "HangSeng_HK_ret_5d", "AMD_ret_1d", "US5Y_Rate_ret_5d", "MSTR_Bitcoin3_ret_1d", "CTAS_Cintas_vol_20d", "ORCL_zscore_60d", "BA_ret_1d", "US3M_Rate_vol_20d", "US6M_Rate_ret_20d", "SJM_JM_Smucker_ret_5d", "PAYX_Paychex_vol_20d"], "is_new": true}, {"model_id": "new_h3_CALM_XGBoost_N15_t6", "algo": "XGBoost", "regime": "CALM", "horizon": 3, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "VRP_ma5", "SJM_JM_Smucker_ret_5d", "Core_CPI_zscore_60d", "MS_MorganStanley_zscore_60d", "NEE_NextEra_ret_20d", "TGT_Target_zscore_60d", "US30Y_Rate_ret_20d", "CPB_CampbellSoup_vol_20d", "EWM_Malaysia_ret_1d", "AMGN_Amgen_ret_1d", "EWM_Malaysia_vol_20d", "XOM_ret_20d", "TXN_vol_20d"], "is_new": true}, {"model_id": "new_h3_CALM_XGBoost_N15_t7", "algo": "XGBoost", "regime": "CALM", "horizon": 3, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "T_ret_1d", "CPB_CampbellSoup_ret_20d", "LUV_SouthwestAir_ret_5d", "BA_ret_1d", "CPB_CampbellSoup_ret_5d", "MSTR_Bitcoin3_ret_20d", "EWY_Korea_zscore_60d", "IWM_SmallCap_vol_20d", "EXC_Exelon_zscore_60d", "NEE_NextEra_ret_20d", "MSTR_Bitcoin3_ret_5d", "EFFR_ret_1d", "MO_AltriaMG_ret_1d"], "is_new": true}, {"model_id": "new_h3_CALM_XGBoost_N20_t0", "algo": "XGBoost", "regime": "CALM", "horizon": 3, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "LMT_LockheedMartin_ret_1d", "EWL_Switzerland_zscore_60d", "NVDA_vol_20d", "CPB_CampbellSoup_zscore_60d", "DAX_Germany_zscore_60d", "ORCL_zscore_60d", "Michigan_Sentiment_ret_20d", "VRP_ma5", "SJM_JM_Smucker_ret_5d", "PAYX_Paychex_zscore_60d", "MS_MorganStanley_zscore_60d", "EOG_EOGResources_vol_20d", "EMR_Emerson_ret_20d", "US7Y_Rate_ret_20d", "LMT_LockheedMartin_vol_20d", "SJM_JM_Smucker_ret_1d", "heston_var_ev_h3", "IBEX_Spain_ret_20d"], "is_new": true}, {"model_id": "new_h3_CALM_XGBoost_N20_t1", "algo": "XGBoost", "regime": "CALM", "horizon": 3, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "US1Y_Rate_ret_5d", "BLK_BlackRock_zscore_60d", "gjr_condvar_h1", "US3M_Rate_vol_20d", "heston_var_ev_h7", "SLB_Schlumberger_ret_1d", "XLV_Health_zscore_60d", "PAYX_Paychex_zscore_60d", "Core_PCE_zscore_60d", "PLD_Prologis_ret_5d", "EWM_Malaysia_zscore_60d", "hmm_p_stress", "EWL_Switzerland_zscore_60d", "CLX_Clorox_vol_20d", "Industrial_Production_zscore_60d", "SCHW_Schwab_ret_5d", "CPB_CampbellSoup_ret_20d", "NWL_Newell_ret_20d"], "is_new": true}, {"model_id": "new_h3_CALM_XGBoost_N20_t2", "algo": "XGBoost", "regime": "CALM", "horizon": 3, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "CI_Cigna_vol_20d", "FedFunds_zscore_60d", "EWY_Korea_zscore_60d", "PAYX_Paychex_vol_20d", "CLX_Clorox_vol_20d", "NOC_Northrop_ret_20d", "IYR_US_REIT2_zscore_60d", "ASX_Australia_vol_20d", "CPB_CampbellSoup_vol_20d", "MO_AltriaMG_ret_1d", "ES_Evergy_ret_1d", "Nikkei_Japan_zscore_60d", "EXC_Exelon_zscore_60d", "HD_ret_20d", "LLY_zscore_60d", "spx_momentum_3d", "SJM_JM_Smucker_ret_5d", "heston_ev_h3"], "is_new": true}, {"model_id": "new_h3_CALM_XGBoost_N20_t3", "algo": "XGBoost", "regime": "CALM", "horizon": 3, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EXC_Exelon_ret_1d", "CLX_Clorox_vol_20d", "spx_momentum_3d", "US3Y_Rate_ret_5d", "Core_CPI_zscore_60d", "EWL_Switzerland_zscore_60d", "PG_ret_20d", "US1Y_Rate_ret_5d", "AXP_Amex_ret_20d", "VVIX_ret_20d", "SLB_Schlumberger_ret_1d", "SBUX_vol_20d", "Nikkei_Japan_vol_20d", "DE_Deere_vol_20d", "AMT_AmericanTower_ret_1d", "XLB_Materials_zscore_60d", "EWA_Australia_ret_1d", "T_ret_1d"], "is_new": true}, {"model_id": "new_h3_CALM_XGBoost_N20_t4", "algo": "XGBoost", "regime": "CALM", "horizon": 3, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "IWM_SmallCap_vol_20d", "ENB_EnbridgeInc_ret_1d", "IBEX_Spain_ret_20d", "EOG_EOGResources_vol_20d", "QQQ_vol_20d", "vix_mean_abs_ret_5d", "NOC_Northrop_ret_20d", "3M_ret_5d", "SBUX_vol_20d", "HD_ret_20d", "spx_momentum_3d", "Core_PCE_zscore_60d", "ORCL_vol_20d", "EXC_Exelon_zscore_60d", "GE_ret_1d", "EOG_EOGResources_ret_5d", "DAX_Germany_vol_20d", "EWA_Australia_ret_1d"], "is_new": true}, {"model_id": "new_h3_CALM_XGBoost_N20_t5", "algo": "XGBoost", "regime": "CALM", "horizon": 3, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "IWM_SmallCap_vol_20d", "LMT_LockheedMartin_vol_20d", "JNJ_ret_1d", "MSTR_Bitcoin3_ret_5d", "CLX_Clorox_vol_20d", "US1Y_Rate_ret_5d", "EWC_Canada_zscore_60d", "DHR_vol_20d", "Nikkei_Japan_zscore_60d", "EFFR_ret_1d", "EWH_HongKong_ret_5d", "TGT_Target_zscore_60d", "EXC_Exelon_ret_1d", "DAX_Germany_vol_20d", "LOW_Lowes_ret_5d", "LLY_zscore_60d", "Brent_Oil_FRED_ret_20d", "AMD_ret_1d"], "is_new": true}, {"model_id": "new_h3_CALM_XGBoost_N20_t6", "algo": "XGBoost", "regime": "CALM", "horizon": 3, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "XOM_ret_20d", "LOW_Lowes_ret_5d", "DHR_vol_20d", "SBUX_ret_5d", "Industrial_Production_zscore_60d", "3M_vol_20d", "MO_AltriaMG_ret_1d", "M_Macys_vol_20d", "TM_Telephone_vol_20d", "SJM_JM_Smucker_ret_5d", "heston_var_ev_h3", "BLK_BlackRock_zscore_60d", "SLB_Schlumberger_ret_5d", "GILD_Gilead_ret_20d", "heston_var_ev_h7", "TED_Spread_vol_20d", "EOG_EOGResources_vol_20d", "AMD_ret_5d"], "is_new": true}, {"model_id": "new_h3_CALM_XGBoost_N20_t7", "algo": "XGBoost", "regime": "CALM", "horizon": 3, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "DOW_Price_zscore_60d", "MS_MorganStanley_ret_5d", "EWA_Australia_ret_1d", "EWG_Germany_vol_20d", "HangSeng_HK_vol_20d", "ASX_Australia_ret_5d", "spx_vol_5d", "ES_Evergy_ret_1d", "spx_momentum_3d", "XLY_Disc_vol_20d", "EWJ_Japan_vol_20d", "SBUX_zscore_60d", "TED_Spread_zscore_60d", "CPB_CampbellSoup_vol_20d", "EWY_Korea_ret_20d", "NEE_NextEra_ret_20d", "PPL_PPL_ret_1d", "EFFR_vol_20d"], "is_new": true}, {"model_id": "new_h3_CALM_XGBoost_N25_t0", "algo": "XGBoost", "regime": "CALM", "horizon": 3, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "NEE_NextEra_ret_20d", "DAX_Germany_zscore_60d", "PAYX_Paychex_vol_20d", "spx_vol_5d", "Nikkei_Japan_vol_20d", "ORCL_zscore_60d", "HD_zscore_60d", "ORCL_vol_20d", "XLV_Health_zscore_60d", "IYR_US_REIT2_zscore_60d", "SLB_Schlumberger_ret_1d", "SLB_Schlumberger_ret_5d", "DAX_Germany_vol_20d", "US6M_Rate_ret_20d", "heston_var_ev_h7", "MSTR_Bitcoin3_ret_5d", "TM_Telephone_ret_1d", "spx_momentum_3d", "PFE_ret_1d", "HUM_Humana_ret_5d", "US7Y_Rate_ret_20d", "DOW_Price_zscore_60d", "US5Y_Rate_ret_5d"], "is_new": true}, {"model_id": "new_h3_CALM_XGBoost_N25_t1", "algo": "XGBoost", "regime": "CALM", "horizon": 3, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWG_Germany_ret_20d", "NOC_Northrop_ret_20d", "SBUX_zscore_60d", "LLY_zscore_60d", "heston_var_ev_h3", "SCHW_Schwab_ret_5d", "EFFR_vol_20d", "Michigan_Sentiment_ret_20d", "NEE_NextEra_ret_20d", "SLB_Schlumberger_ret_1d", "EXC_Exelon_zscore_60d", "XLB_Materials_zscore_60d", "3M_vol_20d", "IYM_BasicMaterials_ret_20d", "EWM_Malaysia_vol_20d", "heston_var_ev_h5", "NWL_Newell_ret_20d", "ORCL_zscore_60d", "SJM_JM_Smucker_ret_5d", "EWH_HongKong_ret_5d", "ES_Evergy_ret_1d", "ASX_Australia_ret_5d", "EWA_Australia_ret_1d"], "is_new": true}, {"model_id": "new_h3_CALM_XGBoost_N25_t2", "algo": "XGBoost", "regime": "CALM", "horizon": 3, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "CPB_CampbellSoup_zscore_60d", "SBUX_ret_5d", "PAYX_Paychex_zscore_60d", "VVIX_ret_20d", "AMD_ret_5d", "JNJ_ret_1d", "EWS_Singapore_ret_5d", "HangSeng_HK_ret_5d", "AVB_AvalonBay_zscore_60d", "EWH_HongKong_ret_5d", "US1Y_Rate_ret_20d", "TGT_Target_zscore_60d", "EWL_Switzerland_zscore_60d", "EWA_Australia_ret_1d", "EWG_Germany_ret_20d", "CPB_CampbellSoup_vol_20d", "INTC_ret_5d", "CPB_CampbellSoup_ret_20d", "SO_SouthernCo_ret_5d", "AMGN_Amgen_ret_1d", "XLK_Tech_zscore_60d", "MS_MorganStanley_zscore_60d", "Michigan_Sentiment_ret_20d"], "is_new": true}, {"model_id": "new_h3_CALM_XGBoost_N25_t3", "algo": "XGBoost", "regime": "CALM", "horizon": 3, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "NWL_Newell_ret_20d", "DE_Deere_vol_20d", "TM_Telephone_ret_1d", "BDX_Becton_Dickinson_ret_20d", "Core_PCE_zscore_60d", "VVIX_ret_20d", "EWL_Switzerland_zscore_60d", "SJM_JM_Smucker_ret_5d", "TED_Spread_zscore_60d", "XLV_Health_zscore_60d", "LUV_SouthwestAir_ret_5d", "EWA_Australia_zscore_60d", "Brent_Oil_FRED_ret_20d", "CTAS_Cintas_vol_20d", "LMT_LockheedMartin_vol_20d", "GILD_Gilead_ret_20d", "EWQ_France_zscore_60d", "US3M_Rate_vol_20d", "SBUX_ret_5d", "TXN_vol_20d", "CPB_CampbellSoup_ret_20d", "LOW_Lowes_ret_5d", "SBUX_vol_20d"], "is_new": true}, {"model_id": "new_h3_CALM_XGBoost_N25_t4", "algo": "XGBoost", "regime": "CALM", "horizon": 3, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "US6M_Rate_ret_20d", "CI_Cigna_vol_20d", "AMT_AmericanTower_ret_1d", "MSTR_Bitcoin3_ret_1d", "Nikkei_Japan_vol_20d", "AMD_ret_5d", "SJM_JM_Smucker_ret_5d", "NVDA_vol_20d", "EQR_Equity_ret_1d", "EWQ_France_ret_20d", "MSTR_Bitcoin3_ret_20d", "LMT_LockheedMartin_vol_20d", "BLK_BlackRock_zscore_60d", "HD_zscore_60d", "MS_MorganStanley_zscore_60d", "ASX_Australia_vol_20d", "DHR_vol_20d", "NFCI_ret_5d", "EOG_EOGResources_vol_20d", "XLF_Fin_vol_20d", "IBEX_Spain_ret_20d", "SCHW_Schwab_ret_5d", "US1Y_Rate_ret_20d"], "is_new": true}, {"model_id": "new_h3_CALM_XGBoost_N25_t5", "algo": "XGBoost", "regime": "CALM", "horizon": 3, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "CPB_CampbellSoup_vol_20d", "heston_var_ev_h3", "T_ret_1d", "DIS_vol_20d", "Nikkei_Japan_vol_20d", "heston_var_ev_h5", "CPB_CampbellSoup_ret_5d", "EWS_Singapore_ret_5d", "Industrial_Production_zscore_60d", "AXP_Amex_ret_20d", "NOC_Northrop_ret_20d", "DAX_Germany_zscore_60d", "BTI_BritishAmerican_ret_20d", "BLK_BlackRock_zscore_60d", "TED_Spread_zscore_60d", "HD_zscore_60d", "ASX_Australia_ret_5d", "Brent_Oil_FRED_ret_5d", "ITT_ITTInc_ret_5d", "Retail_Sales_zscore_60d", "US6M_Rate_ret_20d", "SJM_JM_Smucker_ret_1d", "AMGN_Amgen_ret_1d"], "is_new": true}, {"model_id": "new_h3_CALM_XGBoost_N25_t6", "algo": "XGBoost", "regime": "CALM", "horizon": 3, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "US3M_Rate_vol_20d", "XLK_Tech_zscore_60d", "Retail_Sales_zscore_60d", "heston_var_ev_h5", "LMT_LockheedMartin_vol_20d", "ES_Evergy_ret_1d", "NWL_Newell_ret_20d", "TED_Spread_zscore_60d", "MS_MorganStanley_zscore_60d", "NVDA_vol_20d", "MRK_Merck_zscore_60d", "AXP_Amex_vol_20d", "TGT_Target_zscore_60d", "EWJ_Japan_vol_20d", "heston_ev_h3", "US1Y_Rate_ret_20d", "CMCSA_ret_1d", "MS_MorganStanley_ret_5d", "US1Y_Rate_ret_5d", "EMR_Emerson_ret_20d", "Core_CPI_zscore_60d", "HD_zscore_60d", "AVB_AvalonBay_zscore_60d"], "is_new": true}, {"model_id": "new_h3_CALM_XGBoost_N25_t7", "algo": "XGBoost", "regime": "CALM", "horizon": 3, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "BA_ret_1d", "TED_Spread_vol_20d", "PCAR_PaccarInc_ret_5d", "CI_Cigna_vol_20d", "HD_ret_1d", "EWC_Canada_zscore_60d", "US6M_Rate_ret_20d", "AMGN_Amgen_ret_1d", "EWA_Australia_zscore_60d", "XLY_Disc_vol_20d", "AMD_ret_1d", "XLB_Materials_zscore_60d", "EFFR_ret_1d", "AMZN_ret_5d", "vix_mean_abs_ret_5d", "JNJ_ret_1d", "MS_MorganStanley_ret_5d", "EQIX_Equinix_ret_5d", "MS_MorganStanley_zscore_60d", "AVB_AvalonBay_zscore_60d", "SBUX_ret_5d", "3M_vol_20d", "XOM_ret_20d"], "is_new": true}, {"model_id": "new_h3_CALM_XGBoost_N30_t0", "algo": "XGBoost", "regime": "CALM", "horizon": 3, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "SO_SouthernCo_ret_5d", "IWM_SmallCap_vol_20d", "spx_abs_ret_max_5d", "PAYX_Paychex_vol_20d", "XOM_ret_1d", "T10Y2Y_Spread_ret_5d", "TGT_Target_zscore_60d", "TED_Spread_zscore_60d", "XLV_Health_zscore_60d", "EOG_EOGResources_ret_5d", "BTI_BritishAmerican_ret_20d", "EWM_Malaysia_zscore_60d", "heston_ev_h3", "PG_ret_20d", "EWM_Malaysia_vol_20d", "heston_var_ev_h7", "Core_PCE_zscore_60d", "IYM_BasicMaterials_ret_20d", "EOG_EOGResources_vol_20d", "EFFR_ret_1d", "EWH_HongKong_ret_5d", "WTI_Oil_FRED_zscore_60d", "DE_Deere_vol_20d", "US5Y_Rate_ret_5d", "PCAR_PaccarInc_ret_5d", "VOD_Vodafone_zscore_60d", "vix_acceleration_1d", "EXC_Exelon_ret_1d"], "is_new": true}, {"model_id": "new_h3_CALM_XGBoost_N30_t1", "algo": "XGBoost", "regime": "CALM", "horizon": 3, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "T_ret_1d", "HUM_Humana_ret_5d", "SBUX_zscore_60d", "MO_AltriaMG_ret_1d", "HD_zscore_60d", "EWQ_France_ret_20d", "heston_var_ev_h5", "CI_Cigna_vol_20d", "SLB_Schlumberger_ret_1d", "TM_Telephone_ret_1d", "M_Macys_vol_20d", "IBEX_Spain_ret_20d", "LOW_Lowes_ret_5d", "WTI_Oil_FRED_zscore_60d", "GD_GeneralDynamics_zscore_60d", "PCAR_PaccarInc_ret_5d", "TXN_vol_20d", "BA_ret_1d", "BDX_Becton_Dickinson_ret_20d", "QQQ_vol_20d", "US3Y_Rate_ret_5d", "EOG_EOGResources_vol_20d", "GE_ret_1d", "Nikkei_Japan_vol_20d", "EWL_Switzerland_vol_20d", "gjr_condvar_h1", "NFCI_ret_5d", "T10Y2Y_Spread_ret_5d"], "is_new": true}, {"model_id": "new_h3_CALM_XGBoost_N30_t2", "algo": "XGBoost", "regime": "CALM", "horizon": 3, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "heston_var_ev_h3", "hmm_p_stress", "LUV_SouthwestAir_ret_5d", "heston_var_ev_h7", "AMGN_Amgen_ret_1d", "HD_ret_1d", "Industrial_Production_zscore_60d", "TXN_vol_20d", "EWS_Singapore_ret_5d", "TM_Telephone_vol_20d", "SCHW_Schwab_ret_5d", "LOW_Lowes_ret_5d", "ASX_Australia_vol_20d", "AMD_ret_5d", "SJM_JM_Smucker_ret_1d", "PG_ret_20d", "ASX_Australia_ret_5d", "US6M_Rate_ret_20d", "BA_ret_1d", "EWA_Australia_ret_1d", "Core_CPI_zscore_60d", "PAYX_Paychex_ret_20d", "DIS_vol_20d", "CPB_CampbellSoup_ret_20d", "Brent_Oil_FRED_ret_5d", "EWM_Malaysia_vol_20d", "XOM_ret_20d", "SJM_JM_Smucker_ret_5d"], "is_new": true}, {"model_id": "new_h3_CALM_XGBoost_N30_t3", "algo": "XGBoost", "regime": "CALM", "horizon": 3, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWQ_France_ret_20d", "CTAS_Cintas_vol_20d", "MRK_Merck_zscore_60d", "NVDA_vol_20d", "US6M_Rate_ret_20d", "spx_momentum_3d", "FedFunds_zscore_60d", "AMGN_Amgen_ret_1d", "Core_CPI_zscore_60d", "AXP_Amex_vol_20d", "CI_Cigna_vol_20d", "IBEX_Spain_ret_20d", "Michigan_Sentiment_ret_20d", "AORD_AUS_zscore_60d", "CMCSA_ret_1d", "EWG_Germany_ret_20d", "EWH_HongKong_ret_5d", "IWM_SmallCap_vol_20d", "ES_Evergy_ret_1d", "3M_vol_20d", "ORCL_vol_20d", "ENB_EnbridgeInc_ret_1d", "EOG_EOGResources_vol_20d", "MSTR_Bitcoin3_ret_5d", "BA_ret_1d", "US3M_Rate_zscore_60d", "XLF_Fin_vol_20d", "Brent_Oil_FRED_ret_5d"], "is_new": true}, {"model_id": "new_h3_CALM_XGBoost_N30_t4", "algo": "XGBoost", "regime": "CALM", "horizon": 3, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "M_Macys_vol_20d", "EWM_Malaysia_vol_20d", "EWM_Malaysia_zscore_60d", "Core_CPI_zscore_60d", "MSTR_Bitcoin3_ret_20d", "CPB_CampbellSoup_ret_20d", "HD_zscore_60d", "BA_ret_1d", "T_ret_1d", "EFFR_ret_1d", "EWA_Australia_ret_1d", "CPB_CampbellSoup_vol_20d", "3M_vol_20d", "BTI_BritishAmerican_ret_20d", "QQQ_vol_20d", "CI_Cigna_vol_20d", "PAYX_Paychex_vol_20d", "Core_PCE_zscore_60d", "US7Y_Rate_ret_20d", "HUM_Humana_ret_5d", "INTC_ret_5d", "PLD_Prologis_ret_5d", "EWY_Korea_zscore_60d", "AXP_Amex_ret_20d", "IYR_US_REIT2_zscore_60d", "CTAS_Cintas_vol_20d", "LOW_Lowes_ret_5d", "AORD_AUS_zscore_60d"], "is_new": true}, {"model_id": "new_h3_CALM_XGBoost_N30_t5", "algo": "XGBoost", "regime": "CALM", "horizon": 3, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "IYR_US_REIT2_zscore_60d", "NFCI_ret_5d", "ORCL_zscore_60d", "EWM_Malaysia_zscore_60d", "heston_var_ev_h3", "LOW_Lowes_ret_20d", "EFFR_vol_20d", "NOC_Northrop_ret_20d", "EWM_Malaysia_ret_1d", "LUV_SouthwestAir_ret_5d", "WTI_Oil_FRED_zscore_60d", "MO_AltriaMG_ret_1d", "CLX_Clorox_vol_20d", "PPL_PPL_ret_1d", "Nikkei_Japan_zscore_60d", "EWQ_France_zscore_60d", "LOW_Lowes_ret_5d", "EQR_Equity_ret_1d", "FedFunds_zscore_60d", "CTAS_Cintas_vol_20d", "XLK_Tech_zscore_60d", "AORD_AUS_zscore_60d", "DOW_Price_zscore_60d", "ES_Evergy_ret_1d", "TED_Spread_vol_20d", "MSTR_Bitcoin3_ret_1d", "Retail_Sales_zscore_60d", "CPB_CampbellSoup_ret_5d"], "is_new": true}, {"model_id": "new_h3_CALM_XGBoost_N30_t6", "algo": "XGBoost", "regime": "CALM", "horizon": 3, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "NWL_Newell_ret_20d", "AMT_AmericanTower_ret_1d", "EOG_EOGResources_ret_5d", "HUM_Humana_ret_5d", "US6M_Rate_ret_20d", "ENB_EnbridgeInc_ret_1d", "EWA_Australia_zscore_60d", "LUV_SouthwestAir_ret_5d", "PAYX_Paychex_vol_20d", "ORCL_zscore_60d", "PAYX_Paychex_ret_20d", "HD_ret_5d", "CCI_CrownCastle_vol_20d", "HangSeng_HK_ret_5d", "spx_abs_ret_max_5d", "AXP_Amex_ret_20d", "MO_AltriaMG_ret_1d", "PPL_PPL_ret_1d", "CI_Cigna_vol_20d", "SJM_JM_Smucker_ret_1d", "Industrial_Production_zscore_60d", "heston_var_ev_h7", "DOW_Price_zscore_60d", "DAX_Germany_zscore_60d", "SCHW_Schwab_ret_5d", "US5Y_Rate_ret_5d", "LMT_LockheedMartin_ret_1d", "EMR_Emerson_ret_20d"], "is_new": true}, {"model_id": "new_h3_CALM_XGBoost_N30_t7", "algo": "XGBoost", "regime": "CALM", "horizon": 3, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "T10Y2Y_Spread_ret_5d", "XOM_ret_1d", "TED_Spread_vol_20d", "VVIX_ret_20d", "Core_CPI_zscore_60d", "M_Macys_vol_20d", "US1Y_Rate_ret_20d", "EWY_Korea_ret_20d", "CPB_CampbellSoup_ret_5d", "Retail_Sales_zscore_60d", "CPB_CampbellSoup_zscore_60d", "INTC_ret_5d", "spx_abs_ret_max_5d", "CMCSA_ret_1d", "BTI_BritishAmerican_ret_5d", "EXC_Exelon_zscore_60d", "EMR_Emerson_ret_20d", "FedFunds_zscore_60d", "vix_acceleration_1d", "EFFR_vol_20d", "HD_zscore_60d", "SJM_JM_Smucker_ret_5d", "IBEX_Spain_ret_20d", "LUV_SouthwestAir_ret_5d", "HangSeng_HK_vol_20d", "NOC_Northrop_ret_20d", "ORCL_zscore_60d", "EWL_Switzerland_vol_20d"], "is_new": true}, {"model_id": "new_h3_CALM_LightGBM_N5_t0", "algo": "LightGBM", "regime": "CALM", "horizon": 3, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "PPL_PPL_ret_1d", "FedFunds_zscore_60d", "CMCSA_ret_1d"], "is_new": true}, {"model_id": "new_h3_CALM_LightGBM_N5_t1", "algo": "LightGBM", "regime": "CALM", "horizon": 3, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "VVIX_ret_20d", "ITT_ITTInc_ret_5d", "IBEX_Spain_ret_20d"], "is_new": true}, {"model_id": "new_h3_CALM_LightGBM_N5_t2", "algo": "LightGBM", "regime": "CALM", "horizon": 3, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "3M_vol_20d", "EWA_Australia_zscore_60d", "EXC_Exelon_ret_1d"], "is_new": true}, {"model_id": "new_h3_CALM_LightGBM_N5_t3", "algo": "LightGBM", "regime": "CALM", "horizon": 3, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "Michigan_Sentiment_ret_20d", "CLX_Clorox_vol_20d", "PG_ret_20d"], "is_new": true}, {"model_id": "new_h3_CALM_LightGBM_N5_t4", "algo": "LightGBM", "regime": "CALM", "horizon": 3, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "hmm_p_stress", "ASX_Australia_vol_20d", "Core_CPI_zscore_60d"], "is_new": true}, {"model_id": "new_h3_CALM_LightGBM_N5_t5", "algo": "LightGBM", "regime": "CALM", "horizon": 3, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "TGT_Target_zscore_60d", "EWL_Switzerland_zscore_60d", "CMCSA_ret_1d"], "is_new": true}, {"model_id": "new_h3_CALM_LightGBM_N5_t6", "algo": "LightGBM", "regime": "CALM", "horizon": 3, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "AORD_AUS_zscore_60d", "EWS_Singapore_ret_5d", "INTC_ret_5d"], "is_new": true}, {"model_id": "new_h3_CALM_LightGBM_N5_t7", "algo": "LightGBM", "regime": "CALM", "horizon": 3, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EXC_Exelon_zscore_60d", "AMD_ret_5d", "AORD_AUS_zscore_60d"], "is_new": true}, {"model_id": "new_h3_CALM_LightGBM_N8_t0", "algo": "LightGBM", "regime": "CALM", "horizon": 3, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "DE_Deere_ret_5d", "M_Macys_vol_20d", "BDX_Becton_Dickinson_ret_20d", "ORCL_zscore_60d", "US1Y_Rate_ret_20d", "DOW_Price_zscore_60d"], "is_new": true}, {"model_id": "new_h3_CALM_LightGBM_N8_t1", "algo": "LightGBM", "regime": "CALM", "horizon": 3, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "HD_ret_1d", "EWG_Germany_vol_20d", "GE_ret_1d", "EOG_EOGResources_vol_20d", "HangSeng_HK_ret_5d", "spx_vol_5d"], "is_new": true}, {"model_id": "new_h3_CALM_LightGBM_N8_t2", "algo": "LightGBM", "regime": "CALM", "horizon": 3, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "QQQ_vol_20d", "INTC_ret_5d", "HD_zscore_60d", "spx_momentum_3d", "MS_MorganStanley_zscore_60d", "ORCL_vol_20d"], "is_new": true}, {"model_id": "new_h3_CALM_LightGBM_N8_t3", "algo": "LightGBM", "regime": "CALM", "horizon": 3, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWM_Malaysia_vol_20d", "BLK_BlackRock_zscore_60d", "heston_var_ev_h5", "NWL_Newell_ret_20d", "HUM_Humana_ret_5d", "DAX_Germany_zscore_60d"], "is_new": true}, {"model_id": "new_h3_CALM_LightGBM_N8_t4", "algo": "LightGBM", "regime": "CALM", "horizon": 3, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "SLB_Schlumberger_ret_5d", "EXC_Exelon_zscore_60d", "M_Macys_vol_20d", "BDX_Becton_Dickinson_ret_20d", "EWM_Malaysia_ret_1d", "SPY_zscore_60d"], "is_new": true}, {"model_id": "new_h3_CALM_LightGBM_N8_t5", "algo": "LightGBM", "regime": "CALM", "horizon": 3, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "XLV_Health_zscore_60d", "DOW_Price_zscore_60d", "EWQ_France_ret_20d", "HD_ret_1d", "NWL_Newell_ret_20d", "MS_MorganStanley_ret_5d"], "is_new": true}, {"model_id": "new_h3_CALM_LightGBM_N8_t6", "algo": "LightGBM", "regime": "CALM", "horizon": 3, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "HD_ret_5d", "NWL_Newell_ret_20d", "EXC_Exelon_ret_1d", "IWM_SmallCap_vol_20d", "EWA_Australia_ret_1d", "EWM_Malaysia_zscore_60d"], "is_new": true}, {"model_id": "new_h3_CALM_LightGBM_N8_t7", "algo": "LightGBM", "regime": "CALM", "horizon": 3, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "AMD_ret_1d", "Nikkei_Japan_zscore_60d", "ASX_Australia_ret_5d", "ITT_ITTInc_ret_5d", "heston_var_ev_h7", "HD_ret_5d"], "is_new": true}, {"model_id": "new_h3_CALM_LightGBM_N10_t0", "algo": "LightGBM", "regime": "CALM", "horizon": 3, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "spx_abs_ret_max_5d", "CPB_CampbellSoup_vol_20d", "EWM_Malaysia_ret_1d", "heston_var_ev_h5", "HD_zscore_60d", "NWL_Newell_ret_20d", "EWQ_France_zscore_60d", "gjr_condvar_h1"], "is_new": true}, {"model_id": "new_h3_CALM_LightGBM_N10_t1", "algo": "LightGBM", "regime": "CALM", "horizon": 3, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "heston_var_ev_h3", "SBUX_ret_5d", "TM_Telephone_ret_1d", "Nikkei_Japan_vol_20d", "US5Y_Rate_ret_5d", "ITT_ITTInc_ret_5d", "PAYX_Paychex_zscore_60d", "MSTR_Bitcoin3_ret_1d"], "is_new": true}, {"model_id": "new_h3_CALM_LightGBM_N10_t2", "algo": "LightGBM", "regime": "CALM", "horizon": 3, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "GILD_Gilead_ret_20d", "CPB_CampbellSoup_zscore_60d", "ITT_ITTInc_ret_5d", "heston_ev_h3", "HD_ret_1d", "heston_var_ev_h3", "HD_ret_20d", "SBUX_zscore_60d"], "is_new": true}, {"model_id": "new_h3_CALM_LightGBM_N10_t3", "algo": "LightGBM", "regime": "CALM", "horizon": 3, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "GD_GeneralDynamics_zscore_60d", "EWM_Malaysia_ret_1d", "MRK_Merck_zscore_60d", "spx_abs_ret_max_5d", "XLF_Fin_vol_20d", "PPL_PPL_ret_1d", "EWY_Korea_ret_20d", "Nikkei_Japan_zscore_60d"], "is_new": true}, {"model_id": "new_h3_CALM_LightGBM_N10_t4", "algo": "LightGBM", "regime": "CALM", "horizon": 3, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "vix_acceleration_1d", "Nikkei_Japan_vol_20d", "SJM_JM_Smucker_ret_5d", "LOW_Lowes_ret_5d", "EFFR_ret_1d", "EWQ_France_zscore_60d", "Brent_Oil_FRED_ret_5d", "SBUX_ret_5d"], "is_new": true}, {"model_id": "new_h3_CALM_LightGBM_N10_t5", "algo": "LightGBM", "regime": "CALM", "horizon": 3, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "US3M_Rate_vol_20d", "EWJ_Japan_vol_20d", "QQQ_vol_20d", "AMGN_Amgen_ret_1d", "CI_Cigna_vol_20d", "Core_CPI_zscore_60d", "SPY_zscore_60d", "IWM_SmallCap_vol_20d"], "is_new": true}, {"model_id": "new_h3_CALM_LightGBM_N10_t6", "algo": "LightGBM", "regime": "CALM", "horizon": 3, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "heston_var_ev_h5", "AMGN_Amgen_ret_1d", "XLY_Disc_vol_20d", "LOW_Lowes_ret_20d", "EOG_EOGResources_ret_5d", "US3M_Rate_vol_20d", "AMT_AmericanTower_ret_1d", "IWM_SmallCap_vol_20d"], "is_new": true}, {"model_id": "new_h3_CALM_LightGBM_N10_t7", "algo": "LightGBM", "regime": "CALM", "horizon": 3, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "SBUX_ret_5d", "US3Y_Rate_ret_5d", "EFFR_ret_1d", "SJM_JM_Smucker_ret_5d", "BTI_BritishAmerican_ret_5d", "TED_Spread_zscore_60d", "GD_GeneralDynamics_zscore_60d", "PAYX_Paychex_vol_20d"], "is_new": true}, {"model_id": "new_h3_CALM_LightGBM_N12_t0", "algo": "LightGBM", "regime": "CALM", "horizon": 3, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "SLB_Schlumberger_ret_1d", "AXP_Amex_vol_20d", "MS_MorganStanley_zscore_60d", "EWG_Germany_vol_20d", "XLF_Fin_vol_20d", "heston_ev_h3", "EOG_EOGResources_vol_20d", "HD_ret_5d", "Brent_Oil_FRED_ret_5d", "Nikkei_Japan_vol_20d"], "is_new": true}, {"model_id": "new_h3_CALM_LightGBM_N12_t1", "algo": "LightGBM", "regime": "CALM", "horizon": 3, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "GD_GeneralDynamics_zscore_60d", "ASX_Australia_vol_20d", "heston_ev_h3", "MSTR_Bitcoin3_ret_20d", "EQIX_Equinix_ret_5d", "SBUX_vol_20d", "SO_SouthernCo_ret_5d", "heston_var_ev_h7", "XLK_Tech_zscore_60d", "AMZN_ret_5d"], "is_new": true}, {"model_id": "new_h3_CALM_LightGBM_N12_t2", "algo": "LightGBM", "regime": "CALM", "horizon": 3, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "CLX_Clorox_vol_20d", "DAX_Germany_vol_20d", "DOW_Price_zscore_60d", "vix_acceleration_1d", "Core_PCE_zscore_60d", "PFE_ret_1d", "US7Y_Rate_ret_20d", "IYR_US_REIT2_zscore_60d", "XLB_Materials_zscore_60d", "DHR_ret_1d"], "is_new": true}, {"model_id": "new_h3_CALM_LightGBM_N12_t3", "algo": "LightGBM", "regime": "CALM", "horizon": 3, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWQ_France_zscore_60d", "CCI_CrownCastle_vol_20d", "ES_Evergy_ret_1d", "IYM_BasicMaterials_ret_20d", "MS_MorganStanley_ret_1d", "SBUX_vol_20d", "SJM_JM_Smucker_ret_1d", "DHR_ret_1d", "LLY_zscore_60d", "3M_ret_5d"], "is_new": true}, {"model_id": "new_h3_CALM_LightGBM_N12_t4", "algo": "LightGBM", "regime": "CALM", "horizon": 3, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "DAX_Germany_vol_20d", "NWL_Newell_ret_20d", "EOG_EOGResources_vol_20d", "PFE_ret_1d", "Core_CPI_zscore_60d", "EXC_Exelon_zscore_60d", "CMCSA_ret_1d", "EWA_Australia_ret_1d", "HD_ret_1d", "SJM_JM_Smucker_ret_1d"], "is_new": true}, {"model_id": "new_h3_CALM_LightGBM_N12_t5", "algo": "LightGBM", "regime": "CALM", "horizon": 3, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWG_Germany_ret_20d", "INTC_ret_1d", "hmm_p_stress", "SBUX_zscore_60d", "EWM_Malaysia_zscore_60d", "EXC_Exelon_ret_1d", "EQR_Equity_ret_1d", "CPB_CampbellSoup_ret_5d", "IBEX_Spain_ret_20d", "DOW_Price_zscore_60d"], "is_new": true}, {"model_id": "new_h3_CALM_LightGBM_N12_t6", "algo": "LightGBM", "regime": "CALM", "horizon": 3, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "ORCL_zscore_60d", "VVIX_ret_20d", "EWQ_France_zscore_60d", "SBUX_ret_5d", "EXC_Exelon_zscore_60d", "CTAS_Cintas_vol_20d", "MO_AltriaMG_ret_1d", "hmm_p_stress", "SLB_Schlumberger_ret_5d", "PCAR_PaccarInc_ret_5d"], "is_new": true}, {"model_id": "new_h3_CALM_LightGBM_N12_t7", "algo": "LightGBM", "regime": "CALM", "horizon": 3, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "AXP_Amex_vol_20d", "spx_vol_5d", "IYM_BasicMaterials_ret_20d", "US3M_Rate_zscore_60d", "DE_Deere_vol_20d", "EWM_Malaysia_ret_1d", "heston_var_ev_h5", "3M_ret_5d", "US5Y_Rate_ret_5d", "LUV_SouthwestAir_ret_5d"], "is_new": true}, {"model_id": "new_h3_CALM_LightGBM_N15_t0", "algo": "LightGBM", "regime": "CALM", "horizon": 3, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "MS_MorganStanley_zscore_60d", "HUM_Humana_ret_5d", "vix_mean_abs_ret_5d", "EQR_Equity_ret_1d", "SBUX_zscore_60d", "EWH_HongKong_ret_5d", "PFE_ret_1d", "US7Y_Rate_ret_20d", "LMT_LockheedMartin_ret_1d", "PCAR_PaccarInc_ret_5d", "DOW_Price_zscore_60d", "EWL_Switzerland_zscore_60d", "T10Y2Y_Spread_ret_5d"], "is_new": true}, {"model_id": "new_h3_CALM_LightGBM_N15_t1", "algo": "LightGBM", "regime": "CALM", "horizon": 3, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "DAX_Germany_zscore_60d", "US7Y_Rate_ret_20d", "DOW_Price_zscore_60d", "CI_Cigna_vol_20d", "EWQ_France_zscore_60d", "ITT_ITTInc_ret_5d", "CPB_CampbellSoup_vol_20d", "spx_momentum_3d", "heston_ev_h3", "Brent_Oil_FRED_ret_20d", "spx_abs_ret_max_5d", "EWH_HongKong_ret_5d", "AXP_Amex_vol_20d"], "is_new": true}, {"model_id": "new_h3_CALM_LightGBM_N15_t2", "algo": "LightGBM", "regime": "CALM", "horizon": 3, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWA_Australia_ret_1d", "MRK_Merck_zscore_60d", "AXP_Amex_ret_20d", "EQIX_Equinix_ret_5d", "EXC_Exelon_zscore_60d", "Michigan_Sentiment_ret_20d", "US5Y_Rate_ret_5d", "HD_ret_1d", "TED_Spread_vol_20d", "HD_ret_5d", "CPB_CampbellSoup_vol_20d", "EOG_EOGResources_ret_5d", "EWJ_Japan_vol_20d"], "is_new": true}, {"model_id": "new_h3_CALM_LightGBM_N15_t3", "algo": "LightGBM", "regime": "CALM", "horizon": 3, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EOG_EOGResources_ret_5d", "ASX_Australia_ret_5d", "US1Y_Rate_ret_5d", "SBUX_ret_5d", "EWY_Korea_ret_20d", "TED_Spread_vol_20d", "ITT_ITTInc_ret_5d", "QQQ_vol_20d", "BLK_BlackRock_zscore_60d", "US5Y_Rate_ret_5d", "HangSeng_HK_ret_5d", "EWH_HongKong_ret_5d", "US7Y_Rate_ret_20d"], "is_new": true}, {"model_id": "new_h3_CALM_LightGBM_N15_t4", "algo": "LightGBM", "regime": "CALM", "horizon": 3, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "TED_Spread_vol_20d", "LMT_LockheedMartin_vol_20d", "US30Y_Rate_ret_20d", "ORCL_zscore_60d", "SPY_zscore_60d", "EWG_Germany_vol_20d", "Retail_Sales_zscore_60d", "XLY_Disc_vol_20d", "EOG_EOGResources_ret_5d", "DAX_Germany_vol_20d", "EWM_Malaysia_ret_1d", "CTAS_Cintas_vol_20d", "NWL_Newell_ret_20d"], "is_new": true}, {"model_id": "new_h3_CALM_LightGBM_N15_t5", "algo": "LightGBM", "regime": "CALM", "horizon": 3, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "hmm_p_stress", "BA_ret_1d", "SPY_zscore_60d", "3M_vol_20d", "XLY_Disc_vol_20d", "VVIX_ret_20d", "EWH_HongKong_ret_5d", "PAYX_Paychex_vol_20d", "BTI_BritishAmerican_ret_5d", "MRK_Merck_zscore_60d", "CPB_CampbellSoup_zscore_60d", "XOM_ret_1d", "EWM_Malaysia_zscore_60d"], "is_new": true}, {"model_id": "new_h3_CALM_LightGBM_N15_t6", "algo": "LightGBM", "regime": "CALM", "horizon": 3, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "WTI_Oil_FRED_zscore_60d", "TXN_vol_20d", "US30Y_Rate_ret_20d", "EWY_Korea_ret_20d", "TM_Telephone_vol_20d", "3M_ret_5d", "EWC_Canada_zscore_60d", "PAYX_Paychex_ret_20d", "heston_var_ev_h3", "HD_ret_1d", "hmm_p_stress", "ORCL_zscore_60d", "EWM_Malaysia_zscore_60d"], "is_new": true}, {"model_id": "new_h3_CALM_LightGBM_N15_t7", "algo": "LightGBM", "regime": "CALM", "horizon": 3, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWH_HongKong_ret_5d", "PAYX_Paychex_vol_20d", "BLK_BlackRock_zscore_60d", "vix_mean_abs_ret_5d", "EWY_Korea_zscore_60d", "US5Y_Rate_ret_5d", "GD_GeneralDynamics_zscore_60d", "MS_MorganStanley_zscore_60d", "EOG_EOGResources_ret_5d", "HangSeng_HK_vol_20d", "LUV_SouthwestAir_ret_5d", "INTC_ret_1d", "AORD_AUS_zscore_60d"], "is_new": true}, {"model_id": "new_h3_CALM_LightGBM_N20_t0", "algo": "LightGBM", "regime": "CALM", "horizon": 3, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "ITT_ITTInc_ret_5d", "EWM_Malaysia_zscore_60d", "LOW_Lowes_ret_20d", "hmm_p_stress", "SLB_Schlumberger_ret_1d", "SPY_zscore_60d", "DE_Deere_ret_5d", "EWG_Germany_vol_20d", "FedFunds_zscore_60d", "US1Y_Rate_ret_20d", "CPB_CampbellSoup_ret_5d", "vix_acceleration_1d", "DAX_Germany_vol_20d", "SLB_Schlumberger_ret_5d", "AORD_AUS_zscore_60d", "US1Y_Rate_ret_5d", "EQIX_Equinix_ret_5d", "US3M_Rate_vol_20d"], "is_new": true}, {"model_id": "new_h3_CALM_LightGBM_N20_t1", "algo": "LightGBM", "regime": "CALM", "horizon": 3, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "DE_Deere_vol_20d", "TGT_Target_zscore_60d", "HD_ret_1d", "SBUX_zscore_60d", "SBUX_ret_5d", "VRP_ma5", "US1Y_Rate_ret_20d", "ENB_EnbridgeInc_ret_1d", "JNJ_ret_1d", "SO_SouthernCo_ret_5d", "SBUX_vol_20d", "LMT_LockheedMartin_vol_20d", "EWL_Switzerland_zscore_60d", "PG_ret_20d", "BTI_BritishAmerican_ret_5d", "MRK_Merck_zscore_60d", "Nikkei_Japan_vol_20d", "HUM_Humana_ret_5d"], "is_new": true}, {"model_id": "new_h3_CALM_LightGBM_N20_t2", "algo": "LightGBM", "regime": "CALM", "horizon": 3, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "ASX_Australia_ret_5d", "PAYX_Paychex_vol_20d", "AMGN_Amgen_ret_1d", "MSTR_Bitcoin3_ret_20d", "NFCI_ret_5d", "NOC_Northrop_ret_20d", "SCHW_Schwab_ret_5d", "BLK_BlackRock_zscore_60d", "heston_var_ev_h7", "spx_vol_5d", "M_Macys_vol_20d", "EWJ_Japan_vol_20d", "EWM_Malaysia_zscore_60d", "CMCSA_ret_1d", "CPB_CampbellSoup_ret_5d", "US3Y_Rate_ret_5d", "BTI_BritishAmerican_ret_5d", "Nikkei_Japan_zscore_60d"], "is_new": true}, {"model_id": "new_h3_CALM_LightGBM_N20_t3", "algo": "LightGBM", "regime": "CALM", "horizon": 3, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "Industrial_Production_zscore_60d", "EWL_Switzerland_zscore_60d", "AMZN_ret_5d", "VVIX_ret_20d", "EWS_Singapore_ret_5d", "SBUX_zscore_60d", "EWG_Germany_vol_20d", "Core_CPI_zscore_60d", "EWA_Australia_ret_1d", "EXC_Exelon_ret_1d", "BDX_Becton_Dickinson_ret_20d", "EWM_Malaysia_vol_20d", "SBUX_vol_20d", "SCHW_Schwab_ret_5d", "XOM_ret_1d", "Brent_Oil_FRED_ret_5d", "PLD_Prologis_ret_5d", "AMD_ret_1d"], "is_new": true}, {"model_id": "new_h3_CALM_LightGBM_N20_t4", "algo": "LightGBM", "regime": "CALM", "horizon": 3, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "NOC_Northrop_ret_20d", "CPB_CampbellSoup_ret_20d", "TM_Telephone_ret_1d", "US6M_Rate_ret_20d", "DHR_ret_1d", "BA_ret_1d", "AMD_ret_5d", "HD_ret_5d", "MO_AltriaMG_ret_1d", "SPY_zscore_60d", "DE_Deere_ret_5d", "AXP_Amex_ret_20d", "Retail_Sales_zscore_60d", "DHR_vol_20d", "MS_MorganStanley_ret_5d", "EQR_Equity_ret_1d", "GE_ret_1d", "PAYX_Paychex_vol_20d"], "is_new": true}, {"model_id": "new_h3_CALM_LightGBM_N20_t5", "algo": "LightGBM", "regime": "CALM", "horizon": 3, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "Nikkei_Japan_zscore_60d", "US30Y_Rate_ret_20d", "GE_ret_1d", "Retail_Sales_zscore_60d", "EWS_Singapore_ret_5d", "DIS_vol_20d", "LOW_Lowes_ret_5d", "ITT_ITTInc_ret_5d", "BLK_BlackRock_zscore_60d", "NWL_Newell_ret_20d", "IYM_BasicMaterials_ret_20d", "DE_Deere_vol_20d", "US5Y_Rate_ret_5d", "spx_vol_5d", "XLF_Fin_vol_20d", "AMZN_ret_5d", "IYR_US_REIT2_zscore_60d", "EWL_Switzerland_zscore_60d"], "is_new": true}, {"model_id": "new_h3_CALM_LightGBM_N20_t6", "algo": "LightGBM", "regime": "CALM", "horizon": 3, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "AMZN_ret_5d", "PAYX_Paychex_zscore_60d", "EFFR_vol_20d", "ASX_Australia_ret_5d", "XOM_ret_1d", "LLY_zscore_60d", "Retail_Sales_zscore_60d", "T_ret_1d", "spx_vol_5d", "XLB_Materials_zscore_60d", "AMT_AmericanTower_ret_1d", "EWM_Malaysia_zscore_60d", "INTC_ret_1d", "ITT_ITTInc_ret_5d", "AMD_ret_1d", "TM_Telephone_ret_1d", "CPB_CampbellSoup_ret_5d", "US3M_Rate_zscore_60d"], "is_new": true}, {"model_id": "new_h3_CALM_LightGBM_N20_t7", "algo": "LightGBM", "regime": "CALM", "horizon": 3, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "ITT_ITTInc_ret_5d", "ORCL_vol_20d", "IWM_SmallCap_vol_20d", "PAYX_Paychex_vol_20d", "TM_Telephone_ret_1d", "CCI_CrownCastle_vol_20d", "FedFunds_zscore_60d", "HUM_Humana_ret_5d", "ES_Evergy_ret_1d", "ASX_Australia_ret_5d", "INTC_ret_1d", "EXC_Exelon_zscore_60d", "BDX_Becton_Dickinson_ret_20d", "heston_ev_h3", "LOW_Lowes_ret_5d", "CI_Cigna_vol_20d", "MO_AltriaMG_ret_1d", "vix_acceleration_1d"], "is_new": true}, {"model_id": "new_h3_CALM_LightGBM_N25_t0", "algo": "LightGBM", "regime": "CALM", "horizon": 3, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "AMD_ret_1d", "US3M_Rate_vol_20d", "EOG_EOGResources_vol_20d", "AXP_Amex_vol_20d", "Core_CPI_zscore_60d", "MS_MorganStanley_ret_5d", "heston_var_ev_h7", "SJM_JM_Smucker_ret_1d", "INTC_ret_1d", "WTI_Oil_FRED_zscore_60d", "VVIX_ret_20d", "EWS_Singapore_ret_5d", "ASX_Australia_vol_20d", "AMZN_ret_5d", "Brent_Oil_FRED_ret_20d", "EWM_Malaysia_ret_1d", "EXC_Exelon_ret_1d", "CCI_CrownCastle_vol_20d", "HangSeng_HK_ret_5d", "ASX_Australia_ret_5d", "LMT_LockheedMartin_ret_1d", "TM_Telephone_ret_1d", "BA_ret_1d"], "is_new": true}, {"model_id": "new_h3_CALM_LightGBM_N25_t1", "algo": "LightGBM", "regime": "CALM", "horizon": 3, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EXC_Exelon_ret_1d", "GE_ret_1d", "Nikkei_Japan_zscore_60d", "LOW_Lowes_ret_20d", "IBEX_Spain_ret_20d", "SPY_zscore_60d", "vix_mean_abs_ret_5d", "3M_vol_20d", "AXP_Amex_vol_20d", "AMD_ret_5d", "AVB_AvalonBay_zscore_60d", "HUM_Humana_ret_5d", "ASX_Australia_vol_20d", "AMT_AmericanTower_ret_1d", "HD_zscore_60d", "NFCI_ret_5d", "heston_var_ev_h7", "spx_momentum_3d", "EWY_Korea_zscore_60d", "PG_ret_20d", "MSTR_Bitcoin3_ret_5d", "TED_Spread_zscore_60d", "EWJ_Japan_vol_20d"], "is_new": true}, {"model_id": "new_h3_CALM_LightGBM_N25_t2", "algo": "LightGBM", "regime": "CALM", "horizon": 3, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "LOW_Lowes_ret_5d", "NEE_NextEra_ret_20d", "AMGN_Amgen_ret_1d", "HD_ret_20d", "US1Y_Rate_ret_20d", "US6M_Rate_ret_20d", "XLB_Materials_zscore_60d", "DHR_vol_20d", "spx_momentum_3d", "MS_MorganStanley_zscore_60d", "NWL_Newell_ret_20d", "DE_Deere_vol_20d", "INTC_ret_1d", "PAYX_Paychex_ret_20d", "gjr_condvar_h1", "BLK_BlackRock_zscore_60d", "CI_Cigna_vol_20d", "EWA_Australia_zscore_60d", "TED_Spread_zscore_60d", "AVB_AvalonBay_zscore_60d", "PPL_PPL_ret_1d", "IYM_BasicMaterials_ret_20d", "CCI_CrownCastle_vol_20d"], "is_new": true}, {"model_id": "new_h3_CALM_LightGBM_N25_t3", "algo": "LightGBM", "regime": "CALM", "horizon": 3, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "MSTR_Bitcoin3_ret_5d", "LMT_LockheedMartin_vol_20d", "EQIX_Equinix_ret_5d", "spx_momentum_3d", "MRK_Merck_zscore_60d", "BLK_BlackRock_zscore_60d", "US1Y_Rate_ret_5d", "Core_PCE_zscore_60d", "MO_AltriaMG_ret_1d", "US3M_Rate_vol_20d", "IYM_BasicMaterials_ret_20d", "XLB_Materials_zscore_60d", "AMT_AmericanTower_ret_1d", "HangSeng_HK_vol_20d", "EWQ_France_zscore_60d", "CPB_CampbellSoup_zscore_60d", "Industrial_Production_zscore_60d", "XOM_ret_1d", "ORCL_zscore_60d", "Brent_Oil_FRED_ret_20d", "SJM_JM_Smucker_ret_1d", "EXC_Exelon_ret_1d", "VVIX_ret_20d"], "is_new": true}, {"model_id": "new_h3_CALM_LightGBM_N25_t4", "algo": "LightGBM", "regime": "CALM", "horizon": 3, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "SBUX_ret_5d", "spx_vol_5d", "MS_MorganStanley_ret_1d", "EWA_Australia_ret_1d", "CPB_CampbellSoup_ret_5d", "XOM_ret_20d", "AVB_AvalonBay_zscore_60d", "EWM_Malaysia_vol_20d", "EFFR_vol_20d", "EWY_Korea_ret_20d", "DAX_Germany_vol_20d", "US7Y_Rate_ret_20d", "PFE_ret_1d", "M_Macys_vol_20d", "PPL_PPL_ret_1d", "BDX_Becton_Dickinson_ret_20d", "EMR_Emerson_ret_20d", "LOW_Lowes_ret_5d", "HangSeng_HK_vol_20d", "MSTR_Bitcoin3_ret_1d", "spx_abs_ret_max_5d", "NFCI_ret_5d", "SLB_Schlumberger_ret_5d"], "is_new": true}, {"model_id": "new_h3_CALM_LightGBM_N25_t5", "algo": "LightGBM", "regime": "CALM", "horizon": 3, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "VRP_ma5", "NVDA_vol_20d", "HD_ret_20d", "LUV_SouthwestAir_ret_5d", "heston_var_ev_h5", "SLB_Schlumberger_ret_1d", "XLF_Fin_vol_20d", "EQR_Equity_ret_1d", "EWY_Korea_zscore_60d", "HangSeng_HK_vol_20d", "CI_Cigna_vol_20d", "TM_Telephone_vol_20d", "EWM_Malaysia_vol_20d", "ES_Evergy_ret_1d", "SLB_Schlumberger_ret_5d", "BTI_BritishAmerican_ret_5d", "CLX_Clorox_vol_20d", "EWQ_France_zscore_60d", "PAYX_Paychex_ret_20d", "DAX_Germany_zscore_60d", "GE_ret_1d", "HD_ret_1d", "SCHW_Schwab_ret_5d"], "is_new": true}, {"model_id": "new_h3_CALM_LightGBM_N25_t6", "algo": "LightGBM", "regime": "CALM", "horizon": 3, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "US5Y_Rate_ret_5d", "CMCSA_ret_1d", "GD_GeneralDynamics_zscore_60d", "HangSeng_HK_ret_1d", "TED_Spread_zscore_60d", "vix_mean_abs_ret_5d", "HangSeng_HK_vol_20d", "AMT_AmericanTower_ret_1d", "3M_vol_20d", "hmm_p_stress", "IBEX_Spain_ret_20d", "MSTR_Bitcoin3_ret_5d", "Industrial_Production_zscore_60d", "TXN_vol_20d", "EXC_Exelon_ret_1d", "Michigan_Sentiment_ret_20d", "EWQ_France_ret_20d", "US6M_Rate_ret_20d", "EFFR_ret_1d", "HD_zscore_60d", "US30Y_Rate_ret_20d", "SJM_JM_Smucker_ret_5d", "SCHW_Schwab_ret_5d"], "is_new": true}, {"model_id": "new_h3_CALM_LightGBM_N25_t7", "algo": "LightGBM", "regime": "CALM", "horizon": 3, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "Retail_Sales_zscore_60d", "MS_MorganStanley_zscore_60d", "heston_var_ev_h3", "vix_mean_abs_ret_5d", "NVDA_vol_20d", "EXC_Exelon_ret_1d", "US3Y_Rate_ret_5d", "PPL_PPL_ret_1d", "EWG_Germany_vol_20d", "PAYX_Paychex_zscore_60d", "TED_Spread_zscore_60d", "AXP_Amex_ret_20d", "ITT_ITTInc_ret_5d", "EWY_Korea_ret_20d", "GILD_Gilead_ret_20d", "DIS_vol_20d", "LMT_LockheedMartin_ret_1d", "CTAS_Cintas_vol_20d", "US3M_Rate_vol_20d", "CPB_CampbellSoup_ret_5d", "TM_Telephone_ret_1d", "PCAR_PaccarInc_ret_5d", "DAX_Germany_vol_20d"], "is_new": true}, {"model_id": "new_h3_CALM_LightGBM_N30_t0", "algo": "LightGBM", "regime": "CALM", "horizon": 3, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "PAYX_Paychex_ret_20d", "ORCL_vol_20d", "PFE_ret_1d", "Nikkei_Japan_zscore_60d", "NWL_Newell_ret_20d", "M_Macys_vol_20d", "BDX_Becton_Dickinson_ret_20d", "XLK_Tech_zscore_60d", "CCI_CrownCastle_vol_20d", "VOD_Vodafone_zscore_60d", "Core_PCE_zscore_60d", "WTI_Oil_FRED_zscore_60d", "EWA_Australia_ret_1d", "LUV_SouthwestAir_ret_5d", "EMR_Emerson_ret_20d", "MSTR_Bitcoin3_ret_20d", "EWY_Korea_zscore_60d", "DAX_Germany_zscore_60d", "JNJ_ret_1d", "US5Y_Rate_ret_5d", "EWH_HongKong_ret_5d", "CTAS_Cintas_vol_20d", "PAYX_Paychex_vol_20d", "vix_acceleration_1d", "SJM_JM_Smucker_ret_1d", "AMD_ret_5d", "TED_Spread_zscore_60d", "LMT_LockheedMartin_vol_20d"], "is_new": true}, {"model_id": "new_h3_CALM_LightGBM_N30_t1", "algo": "LightGBM", "regime": "CALM", "horizon": 3, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "AMZN_ret_5d", "XOM_ret_20d", "MRK_Merck_zscore_60d", "spx_abs_ret_max_5d", "ORCL_vol_20d", "EWJ_Japan_vol_20d", "VRP_ma5", "NOC_Northrop_ret_20d", "SBUX_vol_20d", "AXP_Amex_ret_20d", "HangSeng_HK_ret_5d", "HD_ret_1d", "PAYX_Paychex_vol_20d", "Retail_Sales_zscore_60d", "SBUX_ret_5d", "CTAS_Cintas_vol_20d", "GE_ret_1d", "ASX_Australia_ret_5d", "LMT_LockheedMartin_vol_20d", "PAYX_Paychex_zscore_60d", "PLD_Prologis_ret_5d", "BDX_Becton_Dickinson_ret_20d", "DE_Deere_vol_20d", "LUV_SouthwestAir_ret_5d", "AMGN_Amgen_ret_1d", "CI_Cigna_vol_20d", "LOW_Lowes_ret_5d", "MO_AltriaMG_ret_1d"], "is_new": true}, {"model_id": "new_h3_CALM_LightGBM_N30_t2", "algo": "LightGBM", "regime": "CALM", "horizon": 3, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "heston_var_ev_h3", "XLY_Disc_vol_20d", "XOM_ret_1d", "TM_Telephone_ret_1d", "ITT_ITTInc_ret_5d", "PFE_ret_1d", "spx_abs_ret_max_5d", "DE_Deere_ret_5d", "MO_AltriaMG_ret_1d", "MS_MorganStanley_ret_1d", "gjr_condvar_h1", "EWL_Switzerland_vol_20d", "SBUX_zscore_60d", "EXC_Exelon_zscore_60d", "GD_GeneralDynamics_zscore_60d", "T_ret_1d", "XLV_Health_zscore_60d", "LOW_Lowes_ret_20d", "Industrial_Production_zscore_60d", "WTI_Oil_FRED_zscore_60d", "QQQ_vol_20d", "heston_ev_h3", "NEE_NextEra_ret_20d", "EWH_HongKong_ret_5d", "EWQ_France_zscore_60d", "VVIX_ret_20d", "GE_ret_1d", "ES_Evergy_ret_1d"], "is_new": true}, {"model_id": "new_h3_CALM_LightGBM_N30_t3", "algo": "LightGBM", "regime": "CALM", "horizon": 3, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWL_Switzerland_vol_20d", "AMD_ret_1d", "PAYX_Paychex_zscore_60d", "EMR_Emerson_ret_20d", "PPL_PPL_ret_1d", "spx_abs_ret_max_5d", "DE_Deere_ret_5d", "TED_Spread_zscore_60d", "XOM_ret_20d", "DHR_vol_20d", "QQQ_vol_20d", "MS_MorganStanley_ret_1d", "US3Y_Rate_ret_5d", "IWM_SmallCap_vol_20d", "SLB_Schlumberger_ret_1d", "US1Y_Rate_ret_5d", "SJM_JM_Smucker_ret_1d", "BLK_BlackRock_zscore_60d", "XLK_Tech_zscore_60d", "US6M_Rate_ret_20d", "GE_ret_1d", "EFFR_vol_20d", "Nikkei_Japan_vol_20d", "Michigan_Sentiment_ret_20d", "AMT_AmericanTower_ret_1d", "EWQ_France_ret_20d", "NEE_NextEra_ret_20d", "NWL_Newell_ret_20d"], "is_new": true}, {"model_id": "new_h3_CALM_LightGBM_N30_t4", "algo": "LightGBM", "regime": "CALM", "horizon": 3, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "ORCL_vol_20d", "HangSeng_HK_vol_20d", "FedFunds_zscore_60d", "TM_Telephone_vol_20d", "EOG_EOGResources_vol_20d", "heston_var_ev_h5", "WTI_Oil_FRED_zscore_60d", "HD_ret_5d", "EWC_Canada_zscore_60d", "LOW_Lowes_ret_5d", "AXP_Amex_ret_20d", "BDX_Becton_Dickinson_ret_20d", "SBUX_ret_5d", "XLB_Materials_zscore_60d", "MS_MorganStanley_zscore_60d", "CPB_CampbellSoup_vol_20d", "Nikkei_Japan_zscore_60d", "LMT_LockheedMartin_vol_20d", "EWM_Malaysia_ret_1d", "Retail_Sales_zscore_60d", "SCHW_Schwab_ret_5d", "PLD_Prologis_ret_5d", "TED_Spread_vol_20d", "EXC_Exelon_ret_1d", "DHR_ret_1d", "HD_ret_1d", "SBUX_zscore_60d", "CPB_CampbellSoup_zscore_60d"], "is_new": true}, {"model_id": "new_h3_CALM_LightGBM_N30_t5", "algo": "LightGBM", "regime": "CALM", "horizon": 3, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "MS_MorganStanley_ret_1d", "EWA_Australia_zscore_60d", "Nikkei_Japan_vol_20d", "US6M_Rate_ret_20d", "IYR_US_REIT2_zscore_60d", "AMT_AmericanTower_ret_1d", "PPL_PPL_ret_1d", "EOG_EOGResources_vol_20d", "JNJ_ret_1d", "XLF_Fin_vol_20d", "EQIX_Equinix_ret_5d", "LLY_zscore_60d", "ASX_Australia_vol_20d", "CPB_CampbellSoup_ret_5d", "vix_mean_abs_ret_5d", "heston_ev_h3", "Brent_Oil_FRED_ret_20d", "LMT_LockheedMartin_ret_1d", "spx_vol_5d", "QQQ_vol_20d", "TED_Spread_zscore_60d", "BLK_BlackRock_zscore_60d", "Retail_Sales_zscore_60d", "SLB_Schlumberger_ret_5d", "DHR_vol_20d", "EWH_HongKong_ret_5d", "BA_ret_1d", "US7Y_Rate_ret_20d"], "is_new": true}, {"model_id": "new_h3_CALM_LightGBM_N30_t6", "algo": "LightGBM", "regime": "CALM", "horizon": 3, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWC_Canada_zscore_60d", "EWL_Switzerland_vol_20d", "HangSeng_HK_vol_20d", "US6M_Rate_ret_20d", "AMGN_Amgen_ret_1d", "DHR_vol_20d", "EWH_HongKong_ret_5d", "Retail_Sales_zscore_60d", "HangSeng_HK_ret_1d", "EWM_Malaysia_zscore_60d", "EQIX_Equinix_ret_5d", "Nikkei_Japan_zscore_60d", "US3Y_Rate_ret_5d", "EOG_EOGResources_vol_20d", "HD_ret_1d", "HD_ret_20d", "CPB_CampbellSoup_ret_5d", "US1Y_Rate_ret_20d", "3M_ret_5d", "CMCSA_ret_1d", "MSTR_Bitcoin3_ret_20d", "US5Y_Rate_ret_5d", "LMT_LockheedMartin_vol_20d", "Core_PCE_zscore_60d", "spx_momentum_3d", "XLY_Disc_vol_20d", "heston_var_ev_h3", "NVDA_vol_20d"], "is_new": true}, {"model_id": "new_h3_CALM_LightGBM_N30_t7", "algo": "LightGBM", "regime": "CALM", "horizon": 3, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "CPB_CampbellSoup_zscore_60d", "TM_Telephone_vol_20d", "EFFR_ret_1d", "US1Y_Rate_ret_20d", "PG_ret_20d", "WTI_Oil_FRED_zscore_60d", "CPB_CampbellSoup_ret_20d", "FedFunds_zscore_60d", "ORCL_vol_20d", "LOW_Lowes_ret_20d", "VRP_ma5", "LMT_LockheedMartin_ret_1d", "heston_var_ev_h7", "CI_Cigna_vol_20d", "DIS_vol_20d", "AORD_AUS_zscore_60d", "EWA_Australia_ret_1d", "EWM_Malaysia_zscore_60d", "US3M_Rate_vol_20d", "BLK_BlackRock_zscore_60d", "MO_AltriaMG_ret_1d", "MSTR_Bitcoin3_ret_5d", "Nikkei_Japan_zscore_60d", "US7Y_Rate_ret_20d", "EWY_Korea_zscore_60d", "spx_vol_5d", "EWL_Switzerland_vol_20d", "HD_zscore_60d"], "is_new": true}, {"model_id": "new_h3_CALM_GradientBoosting_N5_t0", "algo": "GradientBoosting", "regime": "CALM", "horizon": 3, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "HD_ret_5d", "EFFR_vol_20d", "TED_Spread_zscore_60d"], "is_new": true}, {"model_id": "new_h3_CALM_GradientBoosting_N5_t1", "algo": "GradientBoosting", "regime": "CALM", "horizon": 3, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "QQQ_vol_20d", "Retail_Sales_zscore_60d", "EWQ_France_zscore_60d"], "is_new": true}, {"model_id": "new_h3_CALM_GradientBoosting_N5_t2", "algo": "GradientBoosting", "regime": "CALM", "horizon": 3, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "AMD_ret_1d", "EQIX_Equinix_ret_5d", "PAYX_Paychex_vol_20d"], "is_new": true}, {"model_id": "new_h3_CALM_GradientBoosting_N5_t3", "algo": "GradientBoosting", "regime": "CALM", "horizon": 3, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "US5Y_Rate_ret_5d", "EWG_Germany_vol_20d", "Brent_Oil_FRED_ret_5d"], "is_new": true}, {"model_id": "new_h3_CALM_GradientBoosting_N5_t4", "algo": "GradientBoosting", "regime": "CALM", "horizon": 3, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "IYM_BasicMaterials_ret_20d", "LMT_LockheedMartin_vol_20d", "EWC_Canada_zscore_60d"], "is_new": true}, {"model_id": "new_h3_CALM_GradientBoosting_N5_t5", "algo": "GradientBoosting", "regime": "CALM", "horizon": 3, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "HD_ret_1d", "spx_abs_ret_max_5d", "heston_ev_h3"], "is_new": true}, {"model_id": "new_h3_CALM_GradientBoosting_N5_t6", "algo": "GradientBoosting", "regime": "CALM", "horizon": 3, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "MS_MorganStanley_zscore_60d", "US1Y_Rate_ret_5d", "PG_ret_20d"], "is_new": true}, {"model_id": "new_h3_CALM_GradientBoosting_N5_t7", "algo": "GradientBoosting", "regime": "CALM", "horizon": 3, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "VVIX_ret_20d", "CCI_CrownCastle_vol_20d", "EOG_EOGResources_ret_5d"], "is_new": true}, {"model_id": "new_h3_CALM_GradientBoosting_N8_t0", "algo": "GradientBoosting", "regime": "CALM", "horizon": 3, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "SLB_Schlumberger_ret_1d", "SBUX_vol_20d", "Retail_Sales_zscore_60d", "EXC_Exelon_ret_1d", "SJM_JM_Smucker_ret_1d", "TM_Telephone_ret_1d"], "is_new": true}, {"model_id": "new_h3_CALM_GradientBoosting_N8_t1", "algo": "GradientBoosting", "regime": "CALM", "horizon": 3, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "AMGN_Amgen_ret_1d", "PAYX_Paychex_ret_20d", "MSTR_Bitcoin3_ret_20d", "ENB_EnbridgeInc_ret_1d", "VVIX_ret_20d", "MSTR_Bitcoin3_ret_1d"], "is_new": true}, {"model_id": "new_h3_CALM_GradientBoosting_N8_t2", "algo": "GradientBoosting", "regime": "CALM", "horizon": 3, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "HD_ret_20d", "CPB_CampbellSoup_ret_5d", "INTC_ret_5d", "SBUX_vol_20d", "EWL_Switzerland_zscore_60d", "Brent_Oil_FRED_ret_5d"], "is_new": true}, {"model_id": "new_h3_CALM_GradientBoosting_N8_t3", "algo": "GradientBoosting", "regime": "CALM", "horizon": 3, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "HD_ret_5d", "AMT_AmericanTower_ret_1d", "FedFunds_zscore_60d", "Brent_Oil_FRED_ret_5d", "AMD_ret_1d", "PFE_ret_1d"], "is_new": true}, {"model_id": "new_h3_CALM_GradientBoosting_N8_t4", "algo": "GradientBoosting", "regime": "CALM", "horizon": 3, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWY_Korea_ret_20d", "PCAR_PaccarInc_ret_5d", "BDX_Becton_Dickinson_ret_20d", "EQR_Equity_ret_1d", "NWL_Newell_ret_20d", "LUV_SouthwestAir_ret_5d"], "is_new": true}, {"model_id": "new_h3_CALM_GradientBoosting_N8_t5", "algo": "GradientBoosting", "regime": "CALM", "horizon": 3, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWL_Switzerland_zscore_60d", "LOW_Lowes_ret_20d", "INTC_ret_5d", "AORD_AUS_zscore_60d", "CCI_CrownCastle_vol_20d", "US7Y_Rate_ret_20d"], "is_new": true}, {"model_id": "new_h3_CALM_GradientBoosting_N8_t6", "algo": "GradientBoosting", "regime": "CALM", "horizon": 3, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "spx_vol_5d", "CCI_CrownCastle_vol_20d", "AORD_AUS_zscore_60d", "Core_CPI_zscore_60d", "ITT_ITTInc_ret_5d", "SLB_Schlumberger_ret_5d"], "is_new": true}, {"model_id": "new_h3_CALM_GradientBoosting_N8_t7", "algo": "GradientBoosting", "regime": "CALM", "horizon": 3, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWM_Malaysia_ret_1d", "CMCSA_ret_1d", "EWL_Switzerland_zscore_60d", "Retail_Sales_zscore_60d", "BDX_Becton_Dickinson_ret_20d", "AXP_Amex_ret_20d"], "is_new": true}, {"model_id": "new_h3_CALM_GradientBoosting_N10_t0", "algo": "GradientBoosting", "regime": "CALM", "horizon": 3, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "US3M_Rate_zscore_60d", "EWA_Australia_ret_1d", "CMCSA_ret_1d", "PAYX_Paychex_zscore_60d", "LOW_Lowes_ret_5d", "NOC_Northrop_ret_20d", "AORD_AUS_zscore_60d", "SJM_JM_Smucker_ret_1d"], "is_new": true}, {"model_id": "new_h3_CALM_GradientBoosting_N10_t1", "algo": "GradientBoosting", "regime": "CALM", "horizon": 3, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "PAYX_Paychex_ret_20d", "EWJ_Japan_vol_20d", "PCAR_PaccarInc_ret_5d", "EWQ_France_ret_20d", "VVIX_ret_20d", "SBUX_ret_5d", "heston_ev_h3", "spx_momentum_3d"], "is_new": true}, {"model_id": "new_h3_CALM_GradientBoosting_N10_t2", "algo": "GradientBoosting", "regime": "CALM", "horizon": 3, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "spx_vol_5d", "HD_zscore_60d", "US3M_Rate_vol_20d", "EOG_EOGResources_ret_5d", "NFCI_ret_5d", "EFFR_vol_20d", "Brent_Oil_FRED_ret_5d", "US1Y_Rate_ret_20d"], "is_new": true}, {"model_id": "new_h3_CALM_GradientBoosting_N10_t3", "algo": "GradientBoosting", "regime": "CALM", "horizon": 3, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "AMD_ret_5d", "3M_ret_5d", "NOC_Northrop_ret_20d", "Brent_Oil_FRED_ret_5d", "EWH_HongKong_ret_5d", "TED_Spread_vol_20d", "MSTR_Bitcoin3_ret_20d", "IYR_US_REIT2_zscore_60d"], "is_new": true}, {"model_id": "new_h3_CALM_GradientBoosting_N10_t4", "algo": "GradientBoosting", "regime": "CALM", "horizon": 3, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EOG_EOGResources_ret_5d", "hmm_p_stress", "INTC_ret_1d", "CI_Cigna_vol_20d", "GE_ret_1d", "EWY_Korea_ret_20d", "JNJ_ret_1d", "TM_Telephone_vol_20d"], "is_new": true}, {"model_id": "new_h3_CALM_GradientBoosting_N10_t5", "algo": "GradientBoosting", "regime": "CALM", "horizon": 3, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "CI_Cigna_vol_20d", "BTI_BritishAmerican_ret_20d", "US1Y_Rate_ret_5d", "AORD_AUS_zscore_60d", "ASX_Australia_ret_5d", "HangSeng_HK_ret_5d", "Industrial_Production_zscore_60d", "BA_ret_1d"], "is_new": true}, {"model_id": "new_h3_CALM_GradientBoosting_N10_t6", "algo": "GradientBoosting", "regime": "CALM", "horizon": 3, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "CTAS_Cintas_vol_20d", "US3Y_Rate_ret_5d", "IWM_SmallCap_vol_20d", "XLK_Tech_zscore_60d", "XOM_ret_20d", "NEE_NextEra_ret_20d", "BDX_Becton_Dickinson_ret_20d", "DOW_Price_zscore_60d"], "is_new": true}, {"model_id": "new_h3_CALM_GradientBoosting_N10_t7", "algo": "GradientBoosting", "regime": "CALM", "horizon": 3, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "SCHW_Schwab_ret_5d", "SBUX_vol_20d", "XLF_Fin_vol_20d", "ORCL_vol_20d", "NOC_Northrop_ret_20d", "AXP_Amex_vol_20d", "IWM_SmallCap_vol_20d", "EXC_Exelon_ret_1d"], "is_new": true}, {"model_id": "new_h3_CALM_GradientBoosting_N12_t0", "algo": "GradientBoosting", "regime": "CALM", "horizon": 3, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "INTC_ret_5d", "PPL_PPL_ret_1d", "LMT_LockheedMartin_ret_1d", "AMD_ret_1d", "GE_ret_1d", "EOG_EOGResources_ret_5d", "EQIX_Equinix_ret_5d", "IBEX_Spain_ret_20d", "M_Macys_vol_20d", "EXC_Exelon_zscore_60d"], "is_new": true}, {"model_id": "new_h3_CALM_GradientBoosting_N12_t1", "algo": "GradientBoosting", "regime": "CALM", "horizon": 3, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "CTAS_Cintas_vol_20d", "SBUX_zscore_60d", "Industrial_Production_zscore_60d", "EXC_Exelon_zscore_60d", "CPB_CampbellSoup_zscore_60d", "SPY_zscore_60d", "EFFR_vol_20d", "SJM_JM_Smucker_ret_1d", "Core_CPI_zscore_60d", "HangSeng_HK_ret_1d"], "is_new": true}, {"model_id": "new_h3_CALM_GradientBoosting_N12_t2", "algo": "GradientBoosting", "regime": "CALM", "horizon": 3, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "MO_AltriaMG_ret_1d", "US1Y_Rate_ret_5d", "AMT_AmericanTower_ret_1d", "Brent_Oil_FRED_ret_5d", "US3M_Rate_zscore_60d", "SJM_JM_Smucker_ret_1d", "vix_mean_abs_ret_5d", "IYM_BasicMaterials_ret_20d", "LMT_LockheedMartin_vol_20d", "SBUX_zscore_60d"], "is_new": true}, {"model_id": "new_h3_CALM_GradientBoosting_N12_t3", "algo": "GradientBoosting", "regime": "CALM", "horizon": 3, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "T_ret_1d", "MSTR_Bitcoin3_ret_20d", "IWM_SmallCap_vol_20d", "AXP_Amex_ret_20d", "FedFunds_zscore_60d", "GE_ret_1d", "HD_zscore_60d", "CMCSA_ret_1d", "HangSeng_HK_ret_1d", "HD_ret_1d"], "is_new": true}, {"model_id": "new_h3_CALM_GradientBoosting_N12_t4", "algo": "GradientBoosting", "regime": "CALM", "horizon": 3, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "Michigan_Sentiment_ret_20d", "SCHW_Schwab_ret_5d", "gjr_condvar_h1", "AMGN_Amgen_ret_1d", "TED_Spread_zscore_60d", "TM_Telephone_ret_1d", "BLK_BlackRock_zscore_60d", "LLY_zscore_60d", "AXP_Amex_ret_20d", "BTI_BritishAmerican_ret_20d"], "is_new": true}, {"model_id": "new_h3_CALM_GradientBoosting_N12_t5", "algo": "GradientBoosting", "regime": "CALM", "horizon": 3, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "vix_mean_abs_ret_5d", "SO_SouthernCo_ret_5d", "spx_momentum_3d", "NOC_Northrop_ret_20d", "EQIX_Equinix_ret_5d", "NFCI_ret_5d", "SLB_Schlumberger_ret_5d", "EWY_Korea_zscore_60d", "XLY_Disc_vol_20d", "PAYX_Paychex_ret_20d"], "is_new": true}, {"model_id": "new_h3_CALM_GradientBoosting_N12_t6", "algo": "GradientBoosting", "regime": "CALM", "horizon": 3, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "BTI_BritishAmerican_ret_20d", "NFCI_ret_5d", "US3M_Rate_vol_20d", "vix_acceleration_1d", "SLB_Schlumberger_ret_1d", "Core_CPI_zscore_60d", "HUM_Humana_ret_5d", "DOW_Price_zscore_60d", "3M_vol_20d", "PAYX_Paychex_ret_20d"], "is_new": true}, {"model_id": "new_h3_CALM_GradientBoosting_N12_t7", "algo": "GradientBoosting", "regime": "CALM", "horizon": 3, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "AXP_Amex_ret_20d", "Nikkei_Japan_vol_20d", "TGT_Target_zscore_60d", "Brent_Oil_FRED_ret_5d", "FedFunds_zscore_60d", "EWL_Switzerland_zscore_60d", "LOW_Lowes_ret_5d", "US5Y_Rate_ret_5d", "PPL_PPL_ret_1d", "EFFR_ret_1d"], "is_new": true}, {"model_id": "new_h3_CALM_GradientBoosting_N15_t0", "algo": "GradientBoosting", "regime": "CALM", "horizon": 3, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "NFCI_ret_5d", "CPB_CampbellSoup_vol_20d", "AXP_Amex_vol_20d", "INTC_ret_1d", "EWY_Korea_zscore_60d", "DAX_Germany_vol_20d", "PFE_ret_1d", "CI_Cigna_vol_20d", "ITT_ITTInc_ret_5d", "ASX_Australia_vol_20d", "AORD_AUS_zscore_60d", "SCHW_Schwab_ret_5d", "Nikkei_Japan_zscore_60d"], "is_new": true}, {"model_id": "new_h3_CALM_GradientBoosting_N15_t1", "algo": "GradientBoosting", "regime": "CALM", "horizon": 3, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "CTAS_Cintas_vol_20d", "TM_Telephone_ret_1d", "3M_vol_20d", "PAYX_Paychex_vol_20d", "QQQ_vol_20d", "US1Y_Rate_ret_20d", "EWY_Korea_zscore_60d", "EQIX_Equinix_ret_5d", "SLB_Schlumberger_ret_1d", "HD_ret_20d", "MO_AltriaMG_ret_1d", "LOW_Lowes_ret_5d", "PCAR_PaccarInc_ret_5d"], "is_new": true}, {"model_id": "new_h3_CALM_GradientBoosting_N15_t2", "algo": "GradientBoosting", "regime": "CALM", "horizon": 3, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "SBUX_ret_5d", "AXP_Amex_ret_20d", "Nikkei_Japan_zscore_60d", "Nikkei_Japan_vol_20d", "SBUX_vol_20d", "DE_Deere_ret_5d", "CMCSA_ret_1d", "XLB_Materials_zscore_60d", "EFFR_vol_20d", "EQIX_Equinix_ret_5d", "US1Y_Rate_ret_20d", "EWM_Malaysia_zscore_60d", "HangSeng_HK_ret_1d"], "is_new": true}, {"model_id": "new_h3_CALM_GradientBoosting_N15_t3", "algo": "GradientBoosting", "regime": "CALM", "horizon": 3, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "AMD_ret_1d", "heston_ev_h3", "VRP_ma5", "DOW_Price_zscore_60d", "CTAS_Cintas_vol_20d", "LOW_Lowes_ret_5d", "EFFR_vol_20d", "PAYX_Paychex_zscore_60d", "EQIX_Equinix_ret_5d", "LUV_SouthwestAir_ret_5d", "XOM_ret_1d", "SBUX_vol_20d", "TGT_Target_zscore_60d"], "is_new": true}, {"model_id": "new_h3_CALM_GradientBoosting_N15_t4", "algo": "GradientBoosting", "regime": "CALM", "horizon": 3, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "HD_ret_1d", "BA_ret_1d", "TED_Spread_vol_20d", "US5Y_Rate_ret_5d", "CPB_CampbellSoup_vol_20d", "EOG_EOGResources_ret_5d", "AXP_Amex_ret_20d", "LLY_zscore_60d", "VRP_ma5", "US7Y_Rate_ret_20d", "GD_GeneralDynamics_zscore_60d", "HD_ret_5d", "PLD_Prologis_ret_5d"], "is_new": true}, {"model_id": "new_h3_CALM_GradientBoosting_N15_t5", "algo": "GradientBoosting", "regime": "CALM", "horizon": 3, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EQIX_Equinix_ret_5d", "vix_mean_abs_ret_5d", "ENB_EnbridgeInc_ret_1d", "JNJ_ret_1d", "Michigan_Sentiment_ret_20d", "DE_Deere_ret_5d", "BLK_BlackRock_zscore_60d", "TGT_Target_zscore_60d", "EWL_Switzerland_vol_20d", "NOC_Northrop_ret_20d", "CPB_CampbellSoup_ret_20d", "AMD_ret_5d", "US30Y_Rate_ret_20d"], "is_new": true}, {"model_id": "new_h3_CALM_GradientBoosting_N15_t6", "algo": "GradientBoosting", "regime": "CALM", "horizon": 3, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "SBUX_zscore_60d", "spx_momentum_3d", "ASX_Australia_vol_20d", "BTI_BritishAmerican_ret_5d", "Michigan_Sentiment_ret_20d", "hmm_p_stress", "EOG_EOGResources_ret_5d", "AXP_Amex_vol_20d", "spx_abs_ret_max_5d", "MS_MorganStanley_zscore_60d", "EWQ_France_zscore_60d", "PFE_ret_1d", "BLK_BlackRock_zscore_60d"], "is_new": true}, {"model_id": "new_h3_CALM_GradientBoosting_N15_t7", "algo": "GradientBoosting", "regime": "CALM", "horizon": 3, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "XOM_ret_1d", "CPB_CampbellSoup_zscore_60d", "INTC_ret_5d", "DAX_Germany_vol_20d", "MS_MorganStanley_ret_5d", "ORCL_zscore_60d", "MSTR_Bitcoin3_ret_1d", "Brent_Oil_FRED_ret_5d", "ORCL_vol_20d", "NFCI_ret_5d", "DOW_Price_zscore_60d", "US3Y_Rate_ret_5d", "Michigan_Sentiment_ret_20d"], "is_new": true}, {"model_id": "new_h3_CALM_GradientBoosting_N20_t0", "algo": "GradientBoosting", "regime": "CALM", "horizon": 3, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "BA_ret_1d", "DE_Deere_ret_5d", "LOW_Lowes_ret_5d", "EWS_Singapore_ret_5d", "MSTR_Bitcoin3_ret_1d", "SO_SouthernCo_ret_5d", "SBUX_vol_20d", "EWA_Australia_ret_1d", "DHR_vol_20d", "BLK_BlackRock_zscore_60d", "XLY_Disc_vol_20d", "DIS_vol_20d", "EQIX_Equinix_ret_5d", "IWM_SmallCap_vol_20d", "GILD_Gilead_ret_20d", "AXP_Amex_ret_20d", "T_ret_1d", "AMT_AmericanTower_ret_1d"], "is_new": true}, {"model_id": "new_h3_CALM_GradientBoosting_N20_t1", "algo": "GradientBoosting", "regime": "CALM", "horizon": 3, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "US30Y_Rate_ret_20d", "PG_ret_20d", "EWG_Germany_vol_20d", "hmm_p_stress", "SJM_JM_Smucker_ret_1d", "CPB_CampbellSoup_ret_20d", "TM_Telephone_ret_1d", "EWG_Germany_ret_20d", "heston_ev_h3", "HD_ret_5d", "Core_PCE_zscore_60d", "BDX_Becton_Dickinson_ret_20d", "BA_ret_1d", "heston_var_ev_h3", "3M_vol_20d", "ORCL_vol_20d", "MS_MorganStanley_ret_1d", "ITT_ITTInc_ret_5d"], "is_new": true}, {"model_id": "new_h3_CALM_GradientBoosting_N20_t2", "algo": "GradientBoosting", "regime": "CALM", "horizon": 3, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "US3M_Rate_vol_20d", "EWM_Malaysia_zscore_60d", "AMZN_ret_5d", "3M_vol_20d", "GILD_Gilead_ret_20d", "Nikkei_Japan_zscore_60d", "BLK_BlackRock_zscore_60d", "EWM_Malaysia_ret_1d", "INTC_ret_1d", "hmm_p_stress", "NOC_Northrop_ret_20d", "3M_ret_5d", "TGT_Target_zscore_60d", "XLF_Fin_vol_20d", "XOM_ret_20d", "SJM_JM_Smucker_ret_1d", "ASX_Australia_vol_20d", "MSTR_Bitcoin3_ret_5d"], "is_new": true}, {"model_id": "new_h3_CALM_GradientBoosting_N20_t3", "algo": "GradientBoosting", "regime": "CALM", "horizon": 3, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "XOM_ret_1d", "3M_vol_20d", "HD_ret_20d", "PLD_Prologis_ret_5d", "XLK_Tech_zscore_60d", "BA_ret_1d", "heston_ev_h3", "INTC_ret_1d", "US1Y_Rate_ret_20d", "EWS_Singapore_ret_5d", "BDX_Becton_Dickinson_ret_20d", "FedFunds_zscore_60d", "PPL_PPL_ret_1d", "AMD_ret_5d", "DAX_Germany_zscore_60d", "SBUX_zscore_60d", "spx_abs_ret_max_5d", "TED_Spread_zscore_60d"], "is_new": true}, {"model_id": "new_h3_CALM_GradientBoosting_N20_t4", "algo": "GradientBoosting", "regime": "CALM", "horizon": 3, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "NWL_Newell_ret_20d", "FedFunds_zscore_60d", "MO_AltriaMG_ret_1d", "Core_CPI_zscore_60d", "EWA_Australia_ret_1d", "3M_vol_20d", "AXP_Amex_vol_20d", "ITT_ITTInc_ret_5d", "gjr_condvar_h1", "Core_PCE_zscore_60d", "XOM_ret_1d", "TXN_vol_20d", "IWM_SmallCap_vol_20d", "SBUX_zscore_60d", "IYM_BasicMaterials_ret_20d", "HangSeng_HK_ret_1d", "SBUX_ret_5d", "EWH_HongKong_ret_5d"], "is_new": true}, {"model_id": "new_h3_CALM_GradientBoosting_N20_t5", "algo": "GradientBoosting", "regime": "CALM", "horizon": 3, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "SBUX_vol_20d", "HangSeng_HK_vol_20d", "AMZN_ret_5d", "MSTR_Bitcoin3_ret_20d", "EFFR_vol_20d", "SLB_Schlumberger_ret_5d", "ENB_EnbridgeInc_ret_1d", "SJM_JM_Smucker_ret_5d", "IWM_SmallCap_vol_20d", "3M_ret_5d", "PAYX_Paychex_ret_20d", "EMR_Emerson_ret_20d", "LLY_zscore_60d", "INTC_ret_1d", "EWM_Malaysia_ret_1d", "ES_Evergy_ret_1d", "AMD_ret_5d", "MO_AltriaMG_ret_1d"], "is_new": true}, {"model_id": "new_h3_CALM_GradientBoosting_N20_t6", "algo": "GradientBoosting", "regime": "CALM", "horizon": 3, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "INTC_ret_5d", "MO_AltriaMG_ret_1d", "BDX_Becton_Dickinson_ret_20d", "Core_CPI_zscore_60d", "EWA_Australia_ret_1d", "IYM_BasicMaterials_ret_20d", "EFFR_vol_20d", "EWY_Korea_zscore_60d", "PLD_Prologis_ret_5d", "CCI_CrownCastle_vol_20d", "CMCSA_ret_1d", "ORCL_zscore_60d", "US1Y_Rate_ret_20d", "SLB_Schlumberger_ret_5d", "QQQ_vol_20d", "LLY_zscore_60d", "AMD_ret_5d", "LOW_Lowes_ret_5d"], "is_new": true}, {"model_id": "new_h3_CALM_GradientBoosting_N20_t7", "algo": "GradientBoosting", "regime": "CALM", "horizon": 3, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "CLX_Clorox_vol_20d", "SO_SouthernCo_ret_5d", "AMZN_ret_5d", "LOW_Lowes_ret_20d", "CMCSA_ret_1d", "HD_zscore_60d", "CPB_CampbellSoup_vol_20d", "DAX_Germany_zscore_60d", "AMT_AmericanTower_ret_1d", "3M_ret_5d", "SCHW_Schwab_ret_5d", "EXC_Exelon_zscore_60d", "gjr_condvar_h1", "AVB_AvalonBay_zscore_60d", "GILD_Gilead_ret_20d", "MRK_Merck_zscore_60d", "BTI_BritishAmerican_ret_5d", "IYM_BasicMaterials_ret_20d"], "is_new": true}, {"model_id": "new_h3_CALM_GradientBoosting_N25_t0", "algo": "GradientBoosting", "regime": "CALM", "horizon": 3, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "QQQ_vol_20d", "NEE_NextEra_ret_20d", "XOM_ret_1d", "DIS_vol_20d", "MSTR_Bitcoin3_ret_20d", "US30Y_Rate_ret_20d", "US3Y_Rate_ret_5d", "US7Y_Rate_ret_20d", "FedFunds_zscore_60d", "PAYX_Paychex_vol_20d", "EWM_Malaysia_ret_1d", "SLB_Schlumberger_ret_5d", "PLD_Prologis_ret_5d", "XLK_Tech_zscore_60d", "EWG_Germany_ret_20d", "AVB_AvalonBay_zscore_60d", "US3M_Rate_zscore_60d", "DOW_Price_zscore_60d", "MO_AltriaMG_ret_1d", "EQIX_Equinix_ret_5d", "EWG_Germany_vol_20d", "AXP_Amex_vol_20d", "MSTR_Bitcoin3_ret_1d"], "is_new": true}, {"model_id": "new_h3_CALM_GradientBoosting_N25_t1", "algo": "GradientBoosting", "regime": "CALM", "horizon": 3, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "NFCI_ret_5d", "AXP_Amex_ret_20d", "ORCL_zscore_60d", "CCI_CrownCastle_vol_20d", "DOW_Price_zscore_60d", "HangSeng_HK_ret_5d", "DAX_Germany_vol_20d", "EWG_Germany_vol_20d", "Retail_Sales_zscore_60d", "heston_var_ev_h5", "XLK_Tech_zscore_60d", "HangSeng_HK_ret_1d", "XLV_Health_zscore_60d", "CLX_Clorox_vol_20d", "DHR_ret_1d", "INTC_ret_1d", "TM_Telephone_ret_1d", "BTI_BritishAmerican_ret_5d", "US3M_Rate_zscore_60d", "EWL_Switzerland_zscore_60d", "Nikkei_Japan_zscore_60d", "MS_MorganStanley_ret_1d", "US6M_Rate_ret_20d"], "is_new": true}, {"model_id": "new_h3_CALM_GradientBoosting_N25_t2", "algo": "GradientBoosting", "regime": "CALM", "horizon": 3, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWH_HongKong_ret_5d", "Nikkei_Japan_vol_20d", "LMT_LockheedMartin_ret_1d", "EWA_Australia_zscore_60d", "CI_Cigna_vol_20d", "HangSeng_HK_vol_20d", "INTC_ret_1d", "VVIX_ret_20d", "HD_ret_1d", "MRK_Merck_zscore_60d", "NVDA_vol_20d", "US1Y_Rate_ret_20d", "Core_PCE_zscore_60d", "MO_AltriaMG_ret_1d", "ASX_Australia_ret_5d", "TXN_vol_20d", "US7Y_Rate_ret_20d", "INTC_ret_5d", "CPB_CampbellSoup_zscore_60d", "HUM_Humana_ret_5d", "MSTR_Bitcoin3_ret_20d", "EWM_Malaysia_zscore_60d", "WTI_Oil_FRED_zscore_60d"], "is_new": true}, {"model_id": "new_h3_CALM_GradientBoosting_N25_t3", "algo": "GradientBoosting", "regime": "CALM", "horizon": 3, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWM_Malaysia_vol_20d", "EWQ_France_zscore_60d", "AXP_Amex_ret_20d", "DAX_Germany_zscore_60d", "EWC_Canada_zscore_60d", "AMD_ret_1d", "HD_ret_1d", "EFFR_ret_1d", "EWH_HongKong_ret_5d", "vix_acceleration_1d", "EWL_Switzerland_zscore_60d", "SPY_zscore_60d", "TED_Spread_vol_20d", "heston_ev_h3", "CPB_CampbellSoup_vol_20d", "US3M_Rate_zscore_60d", "MSTR_Bitcoin3_ret_5d", "3M_vol_20d", "DIS_vol_20d", "IWM_SmallCap_vol_20d", "AMZN_ret_5d", "heston_var_ev_h5", "SCHW_Schwab_ret_5d"], "is_new": true}, {"model_id": "new_h3_CALM_GradientBoosting_N25_t4", "algo": "GradientBoosting", "regime": "CALM", "horizon": 3, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "IBEX_Spain_ret_20d", "EWS_Singapore_ret_5d", "TXN_vol_20d", "ORCL_vol_20d", "MSTR_Bitcoin3_ret_20d", "Nikkei_Japan_vol_20d", "SO_SouthernCo_ret_5d", "HangSeng_HK_ret_5d", "PLD_Prologis_ret_5d", "AVB_AvalonBay_zscore_60d", "SJM_JM_Smucker_ret_1d", "EFFR_ret_1d", "MS_MorganStanley_ret_1d", "heston_ev_h3", "EMR_Emerson_ret_20d", "PG_ret_20d", "INTC_ret_1d", "Retail_Sales_zscore_60d", "SLB_Schlumberger_ret_1d", "NVDA_vol_20d", "hmm_p_stress", "EWG_Germany_ret_20d", "NFCI_ret_5d"], "is_new": true}, {"model_id": "new_h3_CALM_GradientBoosting_N25_t5", "algo": "GradientBoosting", "regime": "CALM", "horizon": 3, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "SBUX_vol_20d", "EWA_Australia_zscore_60d", "TED_Spread_zscore_60d", "BDX_Becton_Dickinson_ret_20d", "US1Y_Rate_ret_5d", "TED_Spread_vol_20d", "DIS_vol_20d", "EWG_Germany_ret_20d", "EWA_Australia_ret_1d", "AXP_Amex_ret_20d", "US3M_Rate_vol_20d", "PFE_ret_1d", "AORD_AUS_zscore_60d", "CMCSA_ret_1d", "XLF_Fin_vol_20d", "US6M_Rate_ret_20d", "TM_Telephone_vol_20d", "EOG_EOGResources_ret_5d", "HUM_Humana_ret_5d", "PG_ret_20d", "SJM_JM_Smucker_ret_1d", "AXP_Amex_vol_20d", "PLD_Prologis_ret_5d"], "is_new": true}, {"model_id": "new_h3_CALM_GradientBoosting_N25_t6", "algo": "GradientBoosting", "regime": "CALM", "horizon": 3, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "CPB_CampbellSoup_vol_20d", "3M_ret_5d", "US3M_Rate_vol_20d", "SLB_Schlumberger_ret_1d", "XOM_ret_20d", "NFCI_ret_5d", "EWA_Australia_ret_1d", "Michigan_Sentiment_ret_20d", "PFE_ret_1d", "TED_Spread_zscore_60d", "EOG_EOGResources_vol_20d", "spx_momentum_3d", "BTI_BritishAmerican_ret_5d", "NWL_Newell_ret_20d", "TXN_vol_20d", "US3Y_Rate_ret_5d", "MS_MorganStanley_ret_1d", "AMD_ret_1d", "AMD_ret_5d", "MSTR_Bitcoin3_ret_1d", "M_Macys_vol_20d", "spx_abs_ret_max_5d", "TED_Spread_vol_20d"], "is_new": true}, {"model_id": "new_h3_CALM_GradientBoosting_N25_t7", "algo": "GradientBoosting", "regime": "CALM", "horizon": 3, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "IBEX_Spain_ret_20d", "EMR_Emerson_ret_20d", "MO_AltriaMG_ret_1d", "EOG_EOGResources_ret_5d", "TXN_vol_20d", "GD_GeneralDynamics_zscore_60d", "MSTR_Bitcoin3_ret_1d", "ITT_ITTInc_ret_5d", "SJM_JM_Smucker_ret_5d", "AMT_AmericanTower_ret_1d", "NWL_Newell_ret_20d", "spx_abs_ret_max_5d", "Brent_Oil_FRED_ret_5d", "PAYX_Paychex_zscore_60d", "VVIX_ret_20d", "DE_Deere_vol_20d", "EFFR_vol_20d", "HD_ret_5d", "CPB_CampbellSoup_ret_5d", "PCAR_PaccarInc_ret_5d", "PAYX_Paychex_ret_20d", "AMD_ret_1d", "EWL_Switzerland_vol_20d"], "is_new": true}, {"model_id": "new_h3_CALM_GradientBoosting_N30_t0", "algo": "GradientBoosting", "regime": "CALM", "horizon": 3, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "FedFunds_zscore_60d", "US30Y_Rate_ret_20d", "3M_vol_20d", "XLY_Disc_vol_20d", "TXN_vol_20d", "LLY_zscore_60d", "EXC_Exelon_zscore_60d", "EXC_Exelon_ret_1d", "TED_Spread_zscore_60d", "PCAR_PaccarInc_ret_5d", "US1Y_Rate_ret_20d", "US3Y_Rate_ret_5d", "AXP_Amex_ret_20d", "GE_ret_1d", "TGT_Target_zscore_60d", "EWY_Korea_ret_20d", "EWA_Australia_zscore_60d", "heston_var_ev_h5", "NOC_Northrop_ret_20d", "INTC_ret_5d", "TM_Telephone_vol_20d", "MSTR_Bitcoin3_ret_1d", "PFE_ret_1d", "DE_Deere_vol_20d", "MS_MorganStanley_ret_1d", "BLK_BlackRock_zscore_60d", "EQR_Equity_ret_1d", "CPB_CampbellSoup_ret_5d"], "is_new": true}, {"model_id": "new_h3_CALM_GradientBoosting_N30_t1", "algo": "GradientBoosting", "regime": "CALM", "horizon": 3, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "NWL_Newell_ret_20d", "NOC_Northrop_ret_20d", "HangSeng_HK_ret_1d", "BDX_Becton_Dickinson_ret_20d", "INTC_ret_5d", "heston_var_ev_h3", "XLF_Fin_vol_20d", "US5Y_Rate_ret_5d", "QQQ_vol_20d", "heston_ev_h3", "HangSeng_HK_ret_5d", "EWL_Switzerland_zscore_60d", "AMZN_ret_5d", "TED_Spread_zscore_60d", "SJM_JM_Smucker_ret_1d", "US1Y_Rate_ret_5d", "AMD_ret_1d", "EWJ_Japan_vol_20d", "PAYX_Paychex_ret_20d", "TED_Spread_vol_20d", "EOG_EOGResources_vol_20d", "IYR_US_REIT2_zscore_60d", "EWQ_France_zscore_60d", "ORCL_zscore_60d", "CPB_CampbellSoup_zscore_60d", "DAX_Germany_zscore_60d", "NVDA_vol_20d", "VVIX_ret_20d"], "is_new": true}, {"model_id": "new_h3_CALM_GradientBoosting_N30_t2", "algo": "GradientBoosting", "regime": "CALM", "horizon": 3, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "CCI_CrownCastle_vol_20d", "IYR_US_REIT2_zscore_60d", "INTC_ret_5d", "SBUX_vol_20d", "TM_Telephone_vol_20d", "heston_ev_h3", "CPB_CampbellSoup_ret_5d", "EWQ_France_ret_20d", "gjr_condvar_h1", "ORCL_zscore_60d", "EWG_Germany_ret_20d", "Brent_Oil_FRED_ret_5d", "HD_ret_20d", "ENB_EnbridgeInc_ret_1d", "Core_PCE_zscore_60d", "Retail_Sales_zscore_60d", "Industrial_Production_zscore_60d", "MS_MorganStanley_ret_1d", "DHR_ret_1d", "GE_ret_1d", "SBUX_ret_5d", "GILD_Gilead_ret_20d", "EWY_Korea_zscore_60d", "EWY_Korea_ret_20d", "EWC_Canada_zscore_60d", "CTAS_Cintas_vol_20d", "SJM_JM_Smucker_ret_1d", "TXN_vol_20d"], "is_new": true}, {"model_id": "new_h3_CALM_GradientBoosting_N30_t3", "algo": "GradientBoosting", "regime": "CALM", "horizon": 3, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "LMT_LockheedMartin_vol_20d", "hmm_p_stress", "XLV_Health_zscore_60d", "Retail_Sales_zscore_60d", "NEE_NextEra_ret_20d", "ORCL_vol_20d", "DE_Deere_vol_20d", "ASX_Australia_ret_5d", "TED_Spread_zscore_60d", "IBEX_Spain_ret_20d", "AMD_ret_5d", "ORCL_zscore_60d", "T10Y2Y_Spread_ret_5d", "GILD_Gilead_ret_20d", "M_Macys_vol_20d", "EWY_Korea_zscore_60d", "gjr_condvar_h1", "Core_CPI_zscore_60d", "IYR_US_REIT2_zscore_60d", "PAYX_Paychex_ret_20d", "PAYX_Paychex_vol_20d", "HUM_Humana_ret_5d", "spx_abs_ret_max_5d", "MSTR_Bitcoin3_ret_5d", "DHR_ret_1d", "XLB_Materials_zscore_60d", "SBUX_vol_20d", "SLB_Schlumberger_ret_5d"], "is_new": true}, {"model_id": "new_h3_CALM_GradientBoosting_N30_t4", "algo": "GradientBoosting", "regime": "CALM", "horizon": 3, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "gjr_condvar_h1", "DAX_Germany_zscore_60d", "INTC_ret_1d", "JNJ_ret_1d", "HangSeng_HK_ret_1d", "DOW_Price_zscore_60d", "FedFunds_zscore_60d", "IYR_US_REIT2_zscore_60d", "US30Y_Rate_ret_20d", "NWL_Newell_ret_20d", "M_Macys_vol_20d", "PCAR_PaccarInc_ret_5d", "EOG_EOGResources_vol_20d", "XLK_Tech_zscore_60d", "EQIX_Equinix_ret_5d", "US3Y_Rate_ret_5d", "XLF_Fin_vol_20d", "BDX_Becton_Dickinson_ret_20d", "PG_ret_20d", "EMR_Emerson_ret_20d", "DE_Deere_vol_20d", "NVDA_vol_20d", "PPL_PPL_ret_1d", "GE_ret_1d", "AXP_Amex_ret_20d", "Nikkei_Japan_zscore_60d", "AMZN_ret_5d", "CPB_CampbellSoup_ret_5d"], "is_new": true}, {"model_id": "new_h3_CALM_GradientBoosting_N30_t5", "algo": "GradientBoosting", "regime": "CALM", "horizon": 3, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "SPY_zscore_60d", "EMR_Emerson_ret_20d", "ORCL_zscore_60d", "EWM_Malaysia_zscore_60d", "US1Y_Rate_ret_5d", "NEE_NextEra_ret_20d", "TM_Telephone_ret_1d", "TED_Spread_vol_20d", "EWL_Switzerland_vol_20d", "EWQ_France_ret_20d", "XLV_Health_zscore_60d", "DE_Deere_vol_20d", "EWG_Germany_ret_20d", "heston_var_ev_h7", "XLF_Fin_vol_20d", "heston_var_ev_h5", "MS_MorganStanley_ret_1d", "DHR_ret_1d", "EFFR_ret_1d", "LOW_Lowes_ret_20d", "EWY_Korea_zscore_60d", "NWL_Newell_ret_20d", "SJM_JM_Smucker_ret_5d", "NVDA_vol_20d", "ES_Evergy_ret_1d", "US3M_Rate_vol_20d", "NFCI_ret_5d", "Nikkei_Japan_zscore_60d"], "is_new": true}, {"model_id": "new_h3_CALM_GradientBoosting_N30_t6", "algo": "GradientBoosting", "regime": "CALM", "horizon": 3, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "M_Macys_vol_20d", "MRK_Merck_zscore_60d", "SJM_JM_Smucker_ret_1d", "GD_GeneralDynamics_zscore_60d", "heston_var_ev_h5", "US1Y_Rate_ret_20d", "BA_ret_1d", "US30Y_Rate_ret_20d", "IBEX_Spain_ret_20d", "EWG_Germany_ret_20d", "SBUX_zscore_60d", "CCI_CrownCastle_vol_20d", "HangSeng_HK_vol_20d", "US6M_Rate_ret_20d", "EMR_Emerson_ret_20d", "NFCI_ret_5d", "DIS_vol_20d", "SBUX_ret_5d", "EWM_Malaysia_zscore_60d", "LUV_SouthwestAir_ret_5d", "3M_vol_20d", "vix_mean_abs_ret_5d", "ASX_Australia_ret_5d", "heston_var_ev_h7", "Brent_Oil_FRED_ret_5d", "DOW_Price_zscore_60d", "Nikkei_Japan_vol_20d", "XLK_Tech_zscore_60d"], "is_new": true}, {"model_id": "new_h3_CALM_GradientBoosting_N30_t7", "algo": "GradientBoosting", "regime": "CALM", "horizon": 3, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "Brent_Oil_FRED_ret_5d", "HD_ret_20d", "XLB_Materials_zscore_60d", "ASX_Australia_vol_20d", "US3M_Rate_zscore_60d", "CI_Cigna_vol_20d", "3M_vol_20d", "AXP_Amex_vol_20d", "VVIX_ret_20d", "CPB_CampbellSoup_zscore_60d", "Nikkei_Japan_zscore_60d", "MS_MorganStanley_ret_5d", "LOW_Lowes_ret_20d", "VRP_ma5", "ITT_ITTInc_ret_5d", "IBEX_Spain_ret_20d", "Industrial_Production_zscore_60d", "XLK_Tech_zscore_60d", "EWY_Korea_ret_20d", "spx_momentum_3d", "BA_ret_1d", "CPB_CampbellSoup_ret_20d", "QQQ_vol_20d", "SPY_zscore_60d", "GD_GeneralDynamics_zscore_60d", "M_Macys_vol_20d", "DE_Deere_ret_5d", "SBUX_zscore_60d"], "is_new": true}, {"model_id": "new_h3_CALM_RandomForest_N5_t0", "algo": "RandomForest", "regime": "CALM", "horizon": 3, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "heston_var_ev_h7", "INTC_ret_1d", "EQR_Equity_ret_1d"], "is_new": true}, {"model_id": "new_h3_CALM_RandomForest_N5_t1", "algo": "RandomForest", "regime": "CALM", "horizon": 3, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWC_Canada_zscore_60d", "BDX_Becton_Dickinson_ret_20d", "SJM_JM_Smucker_ret_1d"], "is_new": true}, {"model_id": "new_h3_CALM_RandomForest_N5_t2", "algo": "RandomForest", "regime": "CALM", "horizon": 3, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "MS_MorganStanley_zscore_60d", "MRK_Merck_zscore_60d", "M_Macys_vol_20d"], "is_new": true}, {"model_id": "new_h3_CALM_RandomForest_N5_t3", "algo": "RandomForest", "regime": "CALM", "horizon": 3, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "BLK_BlackRock_zscore_60d", "XOM_ret_20d", "TGT_Target_zscore_60d"], "is_new": true}, {"model_id": "new_h3_CALM_RandomForest_N5_t4", "algo": "RandomForest", "regime": "CALM", "horizon": 3, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "SCHW_Schwab_ret_5d", "CMCSA_ret_1d", "EWJ_Japan_vol_20d"], "is_new": true}, {"model_id": "new_h3_CALM_RandomForest_N5_t5", "algo": "RandomForest", "regime": "CALM", "horizon": 3, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "LOW_Lowes_ret_20d", "ASX_Australia_ret_5d", "AMT_AmericanTower_ret_1d"], "is_new": true}, {"model_id": "new_h3_CALM_RandomForest_N5_t6", "algo": "RandomForest", "regime": "CALM", "horizon": 3, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "PFE_ret_1d", "NVDA_vol_20d", "EXC_Exelon_ret_1d"], "is_new": true}, {"model_id": "new_h3_CALM_RandomForest_N5_t7", "algo": "RandomForest", "regime": "CALM", "horizon": 3, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "Industrial_Production_zscore_60d", "DHR_ret_1d", "Nikkei_Japan_zscore_60d"], "is_new": true}, {"model_id": "new_h3_CALM_RandomForest_N8_t0", "algo": "RandomForest", "regime": "CALM", "horizon": 3, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWA_Australia_zscore_60d", "HD_ret_20d", "SJM_JM_Smucker_ret_1d", "SBUX_zscore_60d", "FedFunds_zscore_60d", "CCI_CrownCastle_vol_20d"], "is_new": true}, {"model_id": "new_h3_CALM_RandomForest_N8_t1", "algo": "RandomForest", "regime": "CALM", "horizon": 3, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "SLB_Schlumberger_ret_5d", "MRK_Merck_zscore_60d", "US3Y_Rate_ret_5d", "LMT_LockheedMartin_vol_20d", "EWA_Australia_zscore_60d", "T_ret_1d"], "is_new": true}, {"model_id": "new_h3_CALM_RandomForest_N8_t2", "algo": "RandomForest", "regime": "CALM", "horizon": 3, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "WTI_Oil_FRED_zscore_60d", "heston_var_ev_h3", "US30Y_Rate_ret_20d", "SCHW_Schwab_ret_5d", "INTC_ret_5d", "US3M_Rate_vol_20d"], "is_new": true}, {"model_id": "new_h3_CALM_RandomForest_N8_t3", "algo": "RandomForest", "regime": "CALM", "horizon": 3, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "BTI_BritishAmerican_ret_5d", "LLY_zscore_60d", "ITT_ITTInc_ret_5d", "EMR_Emerson_ret_20d", "TED_Spread_zscore_60d", "ENB_EnbridgeInc_ret_1d"], "is_new": true}, {"model_id": "new_h3_CALM_RandomForest_N8_t4", "algo": "RandomForest", "regime": "CALM", "horizon": 3, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "GILD_Gilead_ret_20d", "EWM_Malaysia_zscore_60d", "AMT_AmericanTower_ret_1d", "spx_momentum_3d", "EWY_Korea_ret_20d", "XOM_ret_20d"], "is_new": true}, {"model_id": "new_h3_CALM_RandomForest_N8_t5", "algo": "RandomForest", "regime": "CALM", "horizon": 3, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "HUM_Humana_ret_5d", "heston_var_ev_h5", "HD_ret_5d", "EWM_Malaysia_vol_20d", "DIS_vol_20d", "VRP_ma5"], "is_new": true}, {"model_id": "new_h3_CALM_RandomForest_N8_t6", "algo": "RandomForest", "regime": "CALM", "horizon": 3, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "TED_Spread_vol_20d", "Nikkei_Japan_vol_20d", "SJM_JM_Smucker_ret_1d", "heston_var_ev_h5", "HD_ret_20d", "BA_ret_1d"], "is_new": true}, {"model_id": "new_h3_CALM_RandomForest_N8_t7", "algo": "RandomForest", "regime": "CALM", "horizon": 3, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "FedFunds_zscore_60d", "heston_ev_h3", "MS_MorganStanley_ret_5d", "GILD_Gilead_ret_20d", "XLY_Disc_vol_20d", "SBUX_ret_5d"], "is_new": true}, {"model_id": "new_h3_CALM_RandomForest_N10_t0", "algo": "RandomForest", "regime": "CALM", "horizon": 3, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "US3M_Rate_vol_20d", "AMD_ret_1d", "NEE_NextEra_ret_20d", "PAYX_Paychex_zscore_60d", "XOM_ret_20d", "spx_momentum_3d", "Brent_Oil_FRED_ret_5d", "AMT_AmericanTower_ret_1d"], "is_new": true}, {"model_id": "new_h3_CALM_RandomForest_N10_t1", "algo": "RandomForest", "regime": "CALM", "horizon": 3, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EOG_EOGResources_vol_20d", "XLV_Health_zscore_60d", "SLB_Schlumberger_ret_5d", "ENB_EnbridgeInc_ret_1d", "US6M_Rate_ret_20d", "US3M_Rate_zscore_60d", "EWY_Korea_ret_20d", "BTI_BritishAmerican_ret_20d"], "is_new": true}, {"model_id": "new_h3_CALM_RandomForest_N10_t2", "algo": "RandomForest", "regime": "CALM", "horizon": 3, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "DAX_Germany_zscore_60d", "SJM_JM_Smucker_ret_5d", "NWL_Newell_ret_20d", "LOW_Lowes_ret_20d", "SLB_Schlumberger_ret_1d", "LMT_LockheedMartin_ret_1d", "XLB_Materials_zscore_60d", "PPL_PPL_ret_1d"], "is_new": true}, {"model_id": "new_h3_CALM_RandomForest_N10_t3", "algo": "RandomForest", "regime": "CALM", "horizon": 3, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "3M_ret_5d", "JNJ_ret_1d", "heston_var_ev_h5", "LUV_SouthwestAir_ret_5d", "PPL_PPL_ret_1d", "US3M_Rate_vol_20d", "US3Y_Rate_ret_5d", "US3M_Rate_zscore_60d"], "is_new": true}, {"model_id": "new_h3_CALM_RandomForest_N10_t4", "algo": "RandomForest", "regime": "CALM", "horizon": 3, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "FedFunds_zscore_60d", "LMT_LockheedMartin_ret_1d", "MSTR_Bitcoin3_ret_1d", "Core_CPI_zscore_60d", "hmm_p_stress", "DIS_vol_20d", "EWM_Malaysia_ret_1d", "HD_ret_5d"], "is_new": true}, {"model_id": "new_h3_CALM_RandomForest_N10_t5", "algo": "RandomForest", "regime": "CALM", "horizon": 3, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "PLD_Prologis_ret_5d", "AMZN_ret_5d", "SO_SouthernCo_ret_5d", "AMT_AmericanTower_ret_1d", "NOC_Northrop_ret_20d", "hmm_p_stress", "EWL_Switzerland_vol_20d", "Brent_Oil_FRED_ret_20d"], "is_new": true}, {"model_id": "new_h3_CALM_RandomForest_N10_t6", "algo": "RandomForest", "regime": "CALM", "horizon": 3, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWQ_France_zscore_60d", "ITT_ITTInc_ret_5d", "PAYX_Paychex_zscore_60d", "TM_Telephone_vol_20d", "spx_abs_ret_max_5d", "GD_GeneralDynamics_zscore_60d", "IWM_SmallCap_vol_20d", "XLY_Disc_vol_20d"], "is_new": true}, {"model_id": "new_h3_CALM_RandomForest_N10_t7", "algo": "RandomForest", "regime": "CALM", "horizon": 3, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EFFR_ret_1d", "T10Y2Y_Spread_ret_5d", "EMR_Emerson_ret_20d", "Brent_Oil_FRED_ret_20d", "heston_var_ev_h3", "CPB_CampbellSoup_ret_5d", "EQR_Equity_ret_1d", "MS_MorganStanley_ret_5d"], "is_new": true}, {"model_id": "new_h3_CALM_RandomForest_N12_t0", "algo": "RandomForest", "regime": "CALM", "horizon": 3, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "INTC_ret_5d", "EXC_Exelon_ret_1d", "FedFunds_zscore_60d", "SCHW_Schwab_ret_5d", "US1Y_Rate_ret_20d", "SLB_Schlumberger_ret_1d", "US7Y_Rate_ret_20d", "EWL_Switzerland_vol_20d", "PG_ret_20d", "heston_ev_h3"], "is_new": true}, {"model_id": "new_h3_CALM_RandomForest_N12_t1", "algo": "RandomForest", "regime": "CALM", "horizon": 3, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "MO_AltriaMG_ret_1d", "AORD_AUS_zscore_60d", "PAYX_Paychex_ret_20d", "AMZN_ret_5d", "XOM_ret_1d", "heston_var_ev_h7", "AMT_AmericanTower_ret_1d", "heston_var_ev_h3", "M_Macys_vol_20d", "DE_Deere_ret_5d"], "is_new": true}, {"model_id": "new_h3_CALM_RandomForest_N12_t2", "algo": "RandomForest", "regime": "CALM", "horizon": 3, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "Michigan_Sentiment_ret_20d", "EQIX_Equinix_ret_5d", "EXC_Exelon_ret_1d", "spx_momentum_3d", "EWG_Germany_ret_20d", "DE_Deere_ret_5d", "Nikkei_Japan_vol_20d", "IYR_US_REIT2_zscore_60d", "PAYX_Paychex_zscore_60d", "EFFR_ret_1d"], "is_new": true}, {"model_id": "new_h3_CALM_RandomForest_N12_t3", "algo": "RandomForest", "regime": "CALM", "horizon": 3, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "SBUX_zscore_60d", "AVB_AvalonBay_zscore_60d", "SBUX_ret_5d", "SO_SouthernCo_ret_5d", "XOM_ret_1d", "heston_var_ev_h5", "EWQ_France_zscore_60d", "ORCL_zscore_60d", "XLB_Materials_zscore_60d", "SPY_zscore_60d"], "is_new": true}, {"model_id": "new_h3_CALM_RandomForest_N12_t4", "algo": "RandomForest", "regime": "CALM", "horizon": 3, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "hmm_p_stress", "EWA_Australia_ret_1d", "XLY_Disc_vol_20d", "VOD_Vodafone_zscore_60d", "XOM_ret_1d", "ORCL_vol_20d", "EWA_Australia_zscore_60d", "MS_MorganStanley_ret_1d", "US3M_Rate_vol_20d", "CPB_CampbellSoup_zscore_60d"], "is_new": true}, {"model_id": "new_h3_CALM_RandomForest_N12_t5", "algo": "RandomForest", "regime": "CALM", "horizon": 3, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "GD_GeneralDynamics_zscore_60d", "CPB_CampbellSoup_zscore_60d", "VOD_Vodafone_zscore_60d", "TXN_vol_20d", "EWM_Malaysia_ret_1d", "SLB_Schlumberger_ret_1d", "LOW_Lowes_ret_5d", "PAYX_Paychex_zscore_60d", "US3M_Rate_zscore_60d", "EFFR_ret_1d"], "is_new": true}, {"model_id": "new_h3_CALM_RandomForest_N12_t6", "algo": "RandomForest", "regime": "CALM", "horizon": 3, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "XLB_Materials_zscore_60d", "AMGN_Amgen_ret_1d", "CPB_CampbellSoup_ret_5d", "NVDA_vol_20d", "LMT_LockheedMartin_vol_20d", "CPB_CampbellSoup_vol_20d", "INTC_ret_1d", "US5Y_Rate_ret_5d", "ENB_EnbridgeInc_ret_1d", "SBUX_ret_5d"], "is_new": true}, {"model_id": "new_h3_CALM_RandomForest_N12_t7", "algo": "RandomForest", "regime": "CALM", "horizon": 3, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "LOW_Lowes_ret_5d", "heston_var_ev_h5", "IYM_BasicMaterials_ret_20d", "EWM_Malaysia_ret_1d", "EWA_Australia_zscore_60d", "EWG_Germany_vol_20d", "Brent_Oil_FRED_ret_20d", "heston_var_ev_h7", "PAYX_Paychex_vol_20d", "US3Y_Rate_ret_5d"], "is_new": true}, {"model_id": "new_h3_CALM_RandomForest_N15_t0", "algo": "RandomForest", "regime": "CALM", "horizon": 3, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "SCHW_Schwab_ret_5d", "IYM_BasicMaterials_ret_20d", "MSTR_Bitcoin3_ret_5d", "BTI_BritishAmerican_ret_20d", "XLV_Health_zscore_60d", "NWL_Newell_ret_20d", "CPB_CampbellSoup_ret_20d", "PG_ret_20d", "MSTR_Bitcoin3_ret_20d", "LOW_Lowes_ret_5d", "gjr_condvar_h1", "AXP_Amex_vol_20d", "DHR_vol_20d"], "is_new": true}, {"model_id": "new_h3_CALM_RandomForest_N15_t1", "algo": "RandomForest", "regime": "CALM", "horizon": 3, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EOG_EOGResources_ret_5d", "SJM_JM_Smucker_ret_1d", "TED_Spread_zscore_60d", "NWL_Newell_ret_20d", "HangSeng_HK_vol_20d", "spx_vol_5d", "XLK_Tech_zscore_60d", "US3M_Rate_vol_20d", "LOW_Lowes_ret_20d", "T10Y2Y_Spread_ret_5d", "vix_mean_abs_ret_5d", "PG_ret_20d", "BDX_Becton_Dickinson_ret_20d"], "is_new": true}, {"model_id": "new_h3_CALM_RandomForest_N15_t2", "algo": "RandomForest", "regime": "CALM", "horizon": 3, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "IWM_SmallCap_vol_20d", "EWM_Malaysia_vol_20d", "SO_SouthernCo_ret_5d", "hmm_p_stress", "BA_ret_1d", "NVDA_vol_20d", "EWA_Australia_zscore_60d", "CPB_CampbellSoup_ret_20d", "XLV_Health_zscore_60d", "3M_ret_5d", "IYM_BasicMaterials_ret_20d", "EWL_Switzerland_vol_20d", "SCHW_Schwab_ret_5d"], "is_new": true}, {"model_id": "new_h3_CALM_RandomForest_N15_t3", "algo": "RandomForest", "regime": "CALM", "horizon": 3, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "heston_ev_h3", "US5Y_Rate_ret_5d", "ENB_EnbridgeInc_ret_1d", "SJM_JM_Smucker_ret_1d", "HD_ret_5d", "MSTR_Bitcoin3_ret_5d", "ES_Evergy_ret_1d", "EQIX_Equinix_ret_5d", "MS_MorganStanley_zscore_60d", "SO_SouthernCo_ret_5d", "XLK_Tech_zscore_60d", "QQQ_vol_20d", "INTC_ret_5d"], "is_new": true}, {"model_id": "new_h3_CALM_RandomForest_N15_t4", "algo": "RandomForest", "regime": "CALM", "horizon": 3, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EMR_Emerson_ret_20d", "vix_mean_abs_ret_5d", "EXC_Exelon_zscore_60d", "EWG_Germany_vol_20d", "TM_Telephone_ret_1d", "SBUX_zscore_60d", "DE_Deere_vol_20d", "spx_momentum_3d", "AMZN_ret_5d", "PAYX_Paychex_ret_20d", "IWM_SmallCap_vol_20d", "PAYX_Paychex_vol_20d", "XLB_Materials_zscore_60d"], "is_new": true}, {"model_id": "new_h3_CALM_RandomForest_N15_t5", "algo": "RandomForest", "regime": "CALM", "horizon": 3, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWY_Korea_zscore_60d", "spx_abs_ret_max_5d", "XLF_Fin_vol_20d", "GD_GeneralDynamics_zscore_60d", "Core_CPI_zscore_60d", "VRP_ma5", "US3M_Rate_vol_20d", "MSTR_Bitcoin3_ret_1d", "spx_momentum_3d", "GE_ret_1d", "INTC_ret_1d", "US5Y_Rate_ret_5d", "hmm_p_stress"], "is_new": true}, {"model_id": "new_h3_CALM_RandomForest_N15_t6", "algo": "RandomForest", "regime": "CALM", "horizon": 3, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "US1Y_Rate_ret_20d", "HD_ret_1d", "3M_vol_20d", "US5Y_Rate_ret_5d", "DHR_vol_20d", "SCHW_Schwab_ret_5d", "PAYX_Paychex_vol_20d", "vix_acceleration_1d", "Brent_Oil_FRED_ret_5d", "LMT_LockheedMartin_vol_20d", "PAYX_Paychex_zscore_60d", "EWM_Malaysia_vol_20d", "heston_ev_h3"], "is_new": true}, {"model_id": "new_h3_CALM_RandomForest_N15_t7", "algo": "RandomForest", "regime": "CALM", "horizon": 3, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "BLK_BlackRock_zscore_60d", "ITT_ITTInc_ret_5d", "CTAS_Cintas_vol_20d", "TM_Telephone_vol_20d", "EFFR_ret_1d", "LMT_LockheedMartin_ret_1d", "CLX_Clorox_vol_20d", "heston_var_ev_h7", "Core_PCE_zscore_60d", "AORD_AUS_zscore_60d", "EWA_Australia_zscore_60d", "spx_abs_ret_max_5d", "TGT_Target_zscore_60d"], "is_new": true}, {"model_id": "new_h3_CALM_RandomForest_N20_t0", "algo": "RandomForest", "regime": "CALM", "horizon": 3, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "Retail_Sales_zscore_60d", "LMT_LockheedMartin_vol_20d", "US6M_Rate_ret_20d", "CPB_CampbellSoup_vol_20d", "GILD_Gilead_ret_20d", "HD_ret_1d", "MRK_Merck_zscore_60d", "NOC_Northrop_ret_20d", "Nikkei_Japan_vol_20d", "Brent_Oil_FRED_ret_5d", "XLY_Disc_vol_20d", "NEE_NextEra_ret_20d", "Michigan_Sentiment_ret_20d", "PFE_ret_1d", "MSTR_Bitcoin3_ret_1d", "AMD_ret_1d", "CMCSA_ret_1d", "IYR_US_REIT2_zscore_60d"], "is_new": true}, {"model_id": "new_h3_CALM_RandomForest_N20_t1", "algo": "RandomForest", "regime": "CALM", "horizon": 3, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "3M_ret_5d", "CCI_CrownCastle_vol_20d", "ES_Evergy_ret_1d", "MS_MorganStanley_ret_5d", "spx_abs_ret_max_5d", "MRK_Merck_zscore_60d", "JNJ_ret_1d", "PCAR_PaccarInc_ret_5d", "MSTR_Bitcoin3_ret_1d", "HangSeng_HK_ret_5d", "PG_ret_20d", "MO_AltriaMG_ret_1d", "NVDA_vol_20d", "heston_var_ev_h5", "MS_MorganStanley_zscore_60d", "LOW_Lowes_ret_5d", "PFE_ret_1d", "EWA_Australia_zscore_60d"], "is_new": true}, {"model_id": "new_h3_CALM_RandomForest_N20_t2", "algo": "RandomForest", "regime": "CALM", "horizon": 3, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "HangSeng_HK_ret_5d", "CI_Cigna_vol_20d", "BLK_BlackRock_zscore_60d", "Industrial_Production_zscore_60d", "3M_ret_5d", "Nikkei_Japan_zscore_60d", "T10Y2Y_Spread_ret_5d", "IWM_SmallCap_vol_20d", "XLV_Health_zscore_60d", "AMT_AmericanTower_ret_1d", "DOW_Price_zscore_60d", "EWQ_France_zscore_60d", "EWS_Singapore_ret_5d", "BA_ret_1d", "XLK_Tech_zscore_60d", "HangSeng_HK_ret_1d", "Nikkei_Japan_vol_20d", "ES_Evergy_ret_1d"], "is_new": true}, {"model_id": "new_h3_CALM_RandomForest_N20_t3", "algo": "RandomForest", "regime": "CALM", "horizon": 3, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "NVDA_vol_20d", "CPB_CampbellSoup_ret_5d", "MSTR_Bitcoin3_ret_1d", "IYR_US_REIT2_zscore_60d", "EWL_Switzerland_zscore_60d", "PG_ret_20d", "AMD_ret_5d", "ITT_ITTInc_ret_5d", "EWG_Germany_vol_20d", "HUM_Humana_ret_5d", "ENB_EnbridgeInc_ret_1d", "TXN_vol_20d", "VVIX_ret_20d", "3M_ret_5d", "SBUX_zscore_60d", "TED_Spread_zscore_60d", "BTI_BritishAmerican_ret_5d", "heston_ev_h3"], "is_new": true}, {"model_id": "new_h3_CALM_RandomForest_N20_t4", "algo": "RandomForest", "regime": "CALM", "horizon": 3, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "US5Y_Rate_ret_5d", "hmm_p_stress", "MSTR_Bitcoin3_ret_1d", "CTAS_Cintas_vol_20d", "LMT_LockheedMartin_vol_20d", "BDX_Becton_Dickinson_ret_20d", "Retail_Sales_zscore_60d", "AXP_Amex_vol_20d", "Nikkei_Japan_zscore_60d", "MS_MorganStanley_zscore_60d", "DAX_Germany_vol_20d", "XLB_Materials_zscore_60d", "BA_ret_1d", "EWL_Switzerland_zscore_60d", "PFE_ret_1d", "IYM_BasicMaterials_ret_20d", "BLK_BlackRock_zscore_60d", "GD_GeneralDynamics_zscore_60d"], "is_new": true}, {"model_id": "new_h3_CALM_RandomForest_N20_t5", "algo": "RandomForest", "regime": "CALM", "horizon": 3, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "PAYX_Paychex_ret_20d", "XLF_Fin_vol_20d", "Nikkei_Japan_zscore_60d", "EWG_Germany_ret_20d", "CPB_CampbellSoup_vol_20d", "VVIX_ret_20d", "US5Y_Rate_ret_5d", "HD_ret_20d", "NFCI_ret_5d", "ASX_Australia_ret_5d", "US3M_Rate_zscore_60d", "XOM_ret_1d", "MS_MorganStanley_ret_1d", "AXP_Amex_vol_20d", "T_ret_1d", "PG_ret_20d", "GE_ret_1d", "XLV_Health_zscore_60d"], "is_new": true}, {"model_id": "new_h3_CALM_RandomForest_N20_t6", "algo": "RandomForest", "regime": "CALM", "horizon": 3, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "XLB_Materials_zscore_60d", "SLB_Schlumberger_ret_5d", "EWL_Switzerland_zscore_60d", "AMGN_Amgen_ret_1d", "SO_SouthernCo_ret_5d", "vix_mean_abs_ret_5d", "SBUX_zscore_60d", "TM_Telephone_vol_20d", "SPY_zscore_60d", "IBEX_Spain_ret_20d", "PPL_PPL_ret_1d", "PAYX_Paychex_zscore_60d", "US3M_Rate_vol_20d", "SLB_Schlumberger_ret_1d", "EFFR_ret_1d", "EXC_Exelon_ret_1d", "EWL_Switzerland_vol_20d", "PCAR_PaccarInc_ret_5d"], "is_new": true}, {"model_id": "new_h3_CALM_RandomForest_N20_t7", "algo": "RandomForest", "regime": "CALM", "horizon": 3, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EQIX_Equinix_ret_5d", "PAYX_Paychex_vol_20d", "EWY_Korea_zscore_60d", "SJM_JM_Smucker_ret_5d", "ES_Evergy_ret_1d", "EWM_Malaysia_zscore_60d", "AMD_ret_5d", "ASX_Australia_vol_20d", "US3M_Rate_zscore_60d", "ORCL_zscore_60d", "BTI_BritishAmerican_ret_20d", "IWM_SmallCap_vol_20d", "EXC_Exelon_ret_1d", "SBUX_vol_20d", "IYR_US_REIT2_zscore_60d", "VRP_ma5", "LMT_LockheedMartin_vol_20d", "AMGN_Amgen_ret_1d"], "is_new": true}, {"model_id": "new_h3_CALM_RandomForest_N25_t0", "algo": "RandomForest", "regime": "CALM", "horizon": 3, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "PAYX_Paychex_zscore_60d", "DOW_Price_zscore_60d", "spx_momentum_3d", "CLX_Clorox_vol_20d", "BTI_BritishAmerican_ret_20d", "IWM_SmallCap_vol_20d", "MSTR_Bitcoin3_ret_20d", "PAYX_Paychex_vol_20d", "T10Y2Y_Spread_ret_5d", "spx_abs_ret_max_5d", "GILD_Gilead_ret_20d", "AMGN_Amgen_ret_1d", "TM_Telephone_ret_1d", "EFFR_vol_20d", "TGT_Target_zscore_60d", "EQR_Equity_ret_1d", "HUM_Humana_ret_5d", "Michigan_Sentiment_ret_20d", "HangSeng_HK_vol_20d", "Core_PCE_zscore_60d", "EWQ_France_zscore_60d", "vix_mean_abs_ret_5d", "CCI_CrownCastle_vol_20d"], "is_new": true}, {"model_id": "new_h3_CALM_RandomForest_N25_t1", "algo": "RandomForest", "regime": "CALM", "horizon": 3, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "vix_mean_abs_ret_5d", "heston_var_ev_h3", "ENB_EnbridgeInc_ret_1d", "EWL_Switzerland_vol_20d", "CPB_CampbellSoup_zscore_60d", "MRK_Merck_zscore_60d", "US3M_Rate_zscore_60d", "NEE_NextEra_ret_20d", "vix_acceleration_1d", "EWQ_France_ret_20d", "PPL_PPL_ret_1d", "TM_Telephone_ret_1d", "CI_Cigna_vol_20d", "EWM_Malaysia_ret_1d", "Core_PCE_zscore_60d", "XLK_Tech_zscore_60d", "BTI_BritishAmerican_ret_20d", "LUV_SouthwestAir_ret_5d", "spx_abs_ret_max_5d", "SO_SouthernCo_ret_5d", "Brent_Oil_FRED_ret_5d", "EWQ_France_zscore_60d", "MSTR_Bitcoin3_ret_20d"], "is_new": true}, {"model_id": "new_h3_CALM_RandomForest_N25_t2", "algo": "RandomForest", "regime": "CALM", "horizon": 3, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "ORCL_zscore_60d", "VVIX_ret_20d", "AXP_Amex_vol_20d", "HangSeng_HK_ret_5d", "Nikkei_Japan_zscore_60d", "HangSeng_HK_vol_20d", "HD_ret_20d", "WTI_Oil_FRED_zscore_60d", "FedFunds_zscore_60d", "BLK_BlackRock_zscore_60d", "LLY_zscore_60d", "TXN_vol_20d", "BTI_BritishAmerican_ret_20d", "IYM_BasicMaterials_ret_20d", "CPB_CampbellSoup_ret_5d", "XLB_Materials_zscore_60d", "EWM_Malaysia_ret_1d", "EXC_Exelon_ret_1d", "EFFR_ret_1d", "SCHW_Schwab_ret_5d", "US30Y_Rate_ret_20d", "US3M_Rate_vol_20d", "CMCSA_ret_1d"], "is_new": true}, {"model_id": "new_h3_CALM_RandomForest_N25_t3", "algo": "RandomForest", "regime": "CALM", "horizon": 3, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "TGT_Target_zscore_60d", "VVIX_ret_20d", "BA_ret_1d", "AMT_AmericanTower_ret_1d", "XOM_ret_1d", "SJM_JM_Smucker_ret_5d", "Nikkei_Japan_vol_20d", "AXP_Amex_vol_20d", "heston_var_ev_h3", "NWL_Newell_ret_20d", "T10Y2Y_Spread_ret_5d", "Brent_Oil_FRED_ret_20d", "HD_zscore_60d", "HangSeng_HK_vol_20d", "CCI_CrownCastle_vol_20d", "heston_var_ev_h7", "XLK_Tech_zscore_60d", "MS_MorganStanley_ret_1d", "CPB_CampbellSoup_ret_20d", "CPB_CampbellSoup_vol_20d", "Retail_Sales_zscore_60d", "PLD_Prologis_ret_5d", "DOW_Price_zscore_60d"], "is_new": true}, {"model_id": "new_h3_CALM_RandomForest_N25_t4", "algo": "RandomForest", "regime": "CALM", "horizon": 3, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "NEE_NextEra_ret_20d", "PPL_PPL_ret_1d", "AXP_Amex_vol_20d", "US1Y_Rate_ret_5d", "AORD_AUS_zscore_60d", "JNJ_ret_1d", "SPY_zscore_60d", "LLY_zscore_60d", "US1Y_Rate_ret_20d", "EFFR_ret_1d", "HD_ret_1d", "AMD_ret_1d", "EXC_Exelon_ret_1d", "Retail_Sales_zscore_60d", "SBUX_vol_20d", "VOD_Vodafone_zscore_60d", "ITT_ITTInc_ret_5d", "SBUX_ret_5d", "EQIX_Equinix_ret_5d", "HD_ret_20d", "3M_vol_20d", "ASX_Australia_ret_5d", "EWM_Malaysia_zscore_60d"], "is_new": true}, {"model_id": "new_h3_CALM_RandomForest_N25_t5", "algo": "RandomForest", "regime": "CALM", "horizon": 3, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "BTI_BritishAmerican_ret_20d", "T10Y2Y_Spread_ret_5d", "NFCI_ret_5d", "CMCSA_ret_1d", "PAYX_Paychex_vol_20d", "LLY_zscore_60d", "VOD_Vodafone_zscore_60d", "US1Y_Rate_ret_5d", "XLF_Fin_vol_20d", "ORCL_vol_20d", "EQIX_Equinix_ret_5d", "DOW_Price_zscore_60d", "IBEX_Spain_ret_20d", "XLK_Tech_zscore_60d", "HangSeng_HK_ret_5d", "PLD_Prologis_ret_5d", "NVDA_vol_20d", "FedFunds_zscore_60d", "CPB_CampbellSoup_ret_5d", "GE_ret_1d", "heston_var_ev_h7", "hmm_p_stress", "SLB_Schlumberger_ret_1d"], "is_new": true}, {"model_id": "new_h3_CALM_RandomForest_N25_t6", "algo": "RandomForest", "regime": "CALM", "horizon": 3, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "WTI_Oil_FRED_zscore_60d", "US3Y_Rate_ret_5d", "ASX_Australia_vol_20d", "CI_Cigna_vol_20d", "AMD_ret_1d", "DE_Deere_vol_20d", "Michigan_Sentiment_ret_20d", "EWM_Malaysia_zscore_60d", "EWJ_Japan_vol_20d", "FedFunds_zscore_60d", "XOM_ret_1d", "hmm_p_stress", "DIS_vol_20d", "PAYX_Paychex_ret_20d", "SBUX_vol_20d", "QQQ_vol_20d", "BDX_Becton_Dickinson_ret_20d", "TGT_Target_zscore_60d", "IWM_SmallCap_vol_20d", "US5Y_Rate_ret_5d", "HangSeng_HK_vol_20d", "ENB_EnbridgeInc_ret_1d", "NWL_Newell_ret_20d"], "is_new": true}, {"model_id": "new_h3_CALM_RandomForest_N25_t7", "algo": "RandomForest", "regime": "CALM", "horizon": 3, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "MO_AltriaMG_ret_1d", "FedFunds_zscore_60d", "EWC_Canada_zscore_60d", "HD_ret_1d", "EXC_Exelon_ret_1d", "TM_Telephone_ret_1d", "EQR_Equity_ret_1d", "CMCSA_ret_1d", "EWQ_France_zscore_60d", "T10Y2Y_Spread_ret_5d", "3M_vol_20d", "Core_PCE_zscore_60d", "XOM_ret_20d", "INTC_ret_1d", "Industrial_Production_zscore_60d", "Retail_Sales_zscore_60d", "Michigan_Sentiment_ret_20d", "XLB_Materials_zscore_60d", "GILD_Gilead_ret_20d", "CPB_CampbellSoup_ret_5d", "vix_mean_abs_ret_5d", "CPB_CampbellSoup_vol_20d", "EWH_HongKong_ret_5d"], "is_new": true}, {"model_id": "new_h3_CALM_RandomForest_N30_t0", "algo": "RandomForest", "regime": "CALM", "horizon": 3, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EFFR_vol_20d", "EWG_Germany_vol_20d", "PPL_PPL_ret_1d", "LOW_Lowes_ret_5d", "AMD_ret_5d", "SLB_Schlumberger_ret_1d", "EWL_Switzerland_vol_20d", "XLK_Tech_zscore_60d", "SCHW_Schwab_ret_5d", "EWS_Singapore_ret_5d", "Core_PCE_zscore_60d", "PG_ret_20d", "INTC_ret_5d", "ENB_EnbridgeInc_ret_1d", "XOM_ret_1d", "DOW_Price_zscore_60d", "EWL_Switzerland_zscore_60d", "DIS_vol_20d", "HD_ret_1d", "EWQ_France_ret_20d", "EWY_Korea_zscore_60d", "XLB_Materials_zscore_60d", "CPB_CampbellSoup_ret_20d", "vix_acceleration_1d", "US3M_Rate_zscore_60d", "LMT_LockheedMartin_ret_1d", "SBUX_ret_5d", "Nikkei_Japan_zscore_60d"], "is_new": true}, {"model_id": "new_h3_CALM_RandomForest_N30_t1", "algo": "RandomForest", "regime": "CALM", "horizon": 3, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "SBUX_zscore_60d", "EWC_Canada_zscore_60d", "EWM_Malaysia_ret_1d", "MRK_Merck_zscore_60d", "AMGN_Amgen_ret_1d", "AMD_ret_5d", "NVDA_vol_20d", "EXC_Exelon_zscore_60d", "IWM_SmallCap_vol_20d", "heston_var_ev_h3", "T10Y2Y_Spread_ret_5d", "AXP_Amex_ret_20d", "HangSeng_HK_vol_20d", "SPY_zscore_60d", "EOG_EOGResources_ret_5d", "XLK_Tech_zscore_60d", "HD_zscore_60d", "AVB_AvalonBay_zscore_60d", "QQQ_vol_20d", "TM_Telephone_vol_20d", "EFFR_vol_20d", "XLY_Disc_vol_20d", "MSTR_Bitcoin3_ret_20d", "ES_Evergy_ret_1d", "LUV_SouthwestAir_ret_5d", "LLY_zscore_60d", "gjr_condvar_h1", "ORCL_zscore_60d"], "is_new": true}, {"model_id": "new_h3_CALM_RandomForest_N30_t2", "algo": "RandomForest", "regime": "CALM", "horizon": 3, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "AMT_AmericanTower_ret_1d", "Nikkei_Japan_vol_20d", "EFFR_vol_20d", "gjr_condvar_h1", "heston_var_ev_h3", "LOW_Lowes_ret_5d", "SLB_Schlumberger_ret_1d", "PG_ret_20d", "EWL_Switzerland_vol_20d", "MSTR_Bitcoin3_ret_20d", "AMGN_Amgen_ret_1d", "TED_Spread_vol_20d", "BLK_BlackRock_zscore_60d", "MO_AltriaMG_ret_1d", "LMT_LockheedMartin_vol_20d", "TM_Telephone_ret_1d", "DE_Deere_vol_20d", "XLF_Fin_vol_20d", "NVDA_vol_20d", "DHR_vol_20d", "EWG_Germany_ret_20d", "CI_Cigna_vol_20d", "NEE_NextEra_ret_20d", "PFE_ret_1d", "MRK_Merck_zscore_60d", "vix_mean_abs_ret_5d", "BTI_BritishAmerican_ret_20d", "VOD_Vodafone_zscore_60d"], "is_new": true}, {"model_id": "new_h3_CALM_RandomForest_N30_t3", "algo": "RandomForest", "regime": "CALM", "horizon": 3, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "AMZN_ret_5d", "EWL_Switzerland_vol_20d", "HangSeng_HK_ret_1d", "CPB_CampbellSoup_ret_20d", "HD_ret_5d", "PAYX_Paychex_ret_20d", "Retail_Sales_zscore_60d", "Industrial_Production_zscore_60d", "MSTR_Bitcoin3_ret_20d", "TXN_vol_20d", "SLB_Schlumberger_ret_5d", "NVDA_vol_20d", "EWJ_Japan_vol_20d", "SBUX_vol_20d", "CLX_Clorox_vol_20d", "EWQ_France_ret_20d", "ORCL_zscore_60d", "VRP_ma5", "VOD_Vodafone_zscore_60d", "TM_Telephone_ret_1d", "PPL_PPL_ret_1d", "AMD_ret_1d", "TED_Spread_vol_20d", "AORD_AUS_zscore_60d", "IYR_US_REIT2_zscore_60d", "EWY_Korea_ret_20d", "SLB_Schlumberger_ret_1d", "ES_Evergy_ret_1d"], "is_new": true}, {"model_id": "new_h3_CALM_RandomForest_N30_t4", "algo": "RandomForest", "regime": "CALM", "horizon": 3, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "spx_abs_ret_max_5d", "EWM_Malaysia_ret_1d", "spx_momentum_3d", "XLK_Tech_zscore_60d", "MS_MorganStanley_ret_1d", "IYM_BasicMaterials_ret_20d", "PAYX_Paychex_zscore_60d", "LUV_SouthwestAir_ret_5d", "EWA_Australia_zscore_60d", "SBUX_vol_20d", "vix_mean_abs_ret_5d", "GE_ret_1d", "HD_ret_5d", "IYR_US_REIT2_zscore_60d", "Core_CPI_zscore_60d", "EFFR_vol_20d", "IWM_SmallCap_vol_20d", "MSTR_Bitcoin3_ret_5d", "EFFR_ret_1d", "EOG_EOGResources_ret_5d", "WTI_Oil_FRED_zscore_60d", "AMD_ret_5d", "TM_Telephone_vol_20d", "EWG_Germany_vol_20d", "HD_ret_1d", "NEE_NextEra_ret_20d", "EXC_Exelon_ret_1d", "GD_GeneralDynamics_zscore_60d"], "is_new": true}, {"model_id": "new_h3_CALM_RandomForest_N30_t5", "algo": "RandomForest", "regime": "CALM", "horizon": 3, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "SPY_zscore_60d", "BDX_Becton_Dickinson_ret_20d", "PAYX_Paychex_vol_20d", "Brent_Oil_FRED_ret_5d", "US30Y_Rate_ret_20d", "MSTR_Bitcoin3_ret_1d", "AXP_Amex_ret_20d", "Nikkei_Japan_zscore_60d", "US3M_Rate_zscore_60d", "VRP_ma5", "EWG_Germany_ret_20d", "XLB_Materials_zscore_60d", "WTI_Oil_FRED_zscore_60d", "FedFunds_zscore_60d", "JNJ_ret_1d", "spx_momentum_3d", "ASX_Australia_vol_20d", "MSTR_Bitcoin3_ret_5d", "TED_Spread_zscore_60d", "gjr_condvar_h1", "vix_mean_abs_ret_5d", "DOW_Price_zscore_60d", "DHR_vol_20d", "Brent_Oil_FRED_ret_20d", "MSTR_Bitcoin3_ret_20d", "DIS_vol_20d", "heston_ev_h3", "hmm_p_stress"], "is_new": true}, {"model_id": "new_h3_CALM_RandomForest_N30_t6", "algo": "RandomForest", "regime": "CALM", "horizon": 3, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "SLB_Schlumberger_ret_1d", "MRK_Merck_zscore_60d", "SCHW_Schwab_ret_5d", "CPB_CampbellSoup_zscore_60d", "BDX_Becton_Dickinson_ret_20d", "EWS_Singapore_ret_5d", "NVDA_vol_20d", "EWA_Australia_ret_1d", "heston_ev_h3", "AMD_ret_5d", "LOW_Lowes_ret_5d", "MSTR_Bitcoin3_ret_1d", "WTI_Oil_FRED_zscore_60d", "IYR_US_REIT2_zscore_60d", "NOC_Northrop_ret_20d", "EQIX_Equinix_ret_5d", "INTC_ret_1d", "ASX_Australia_ret_5d", "US6M_Rate_ret_20d", "MSTR_Bitcoin3_ret_5d", "HangSeng_HK_ret_1d", "Brent_Oil_FRED_ret_20d", "DHR_ret_1d", "EWJ_Japan_vol_20d", "INTC_ret_5d", "EOG_EOGResources_ret_5d", "T_ret_1d", "TED_Spread_zscore_60d"], "is_new": true}, {"model_id": "new_h3_CALM_RandomForest_N30_t7", "algo": "RandomForest", "regime": "CALM", "horizon": 3, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "TED_Spread_vol_20d", "BTI_BritishAmerican_ret_5d", "CMCSA_ret_1d", "3M_ret_5d", "PFE_ret_1d", "CPB_CampbellSoup_ret_5d", "HD_ret_20d", "VVIX_ret_20d", "EOG_EOGResources_vol_20d", "EWG_Germany_vol_20d", "M_Macys_vol_20d", "AMZN_ret_5d", "EWS_Singapore_ret_5d", "NFCI_ret_5d", "SLB_Schlumberger_ret_5d", "AORD_AUS_zscore_60d", "SLB_Schlumberger_ret_1d", "IYR_US_REIT2_zscore_60d", "US1Y_Rate_ret_5d", "XLF_Fin_vol_20d", "HD_ret_5d", "EXC_Exelon_zscore_60d", "GE_ret_1d", "EWH_HongKong_ret_5d", "SBUX_vol_20d", "DE_Deere_vol_20d", "CTAS_Cintas_vol_20d", "EWC_Canada_zscore_60d"], "is_new": true}, {"model_id": "new_h3_CALM_LogisticRegression_N5_t0", "algo": "LogisticRegression", "regime": "CALM", "horizon": 3, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "NOC_Northrop_ret_20d", "EWJ_Japan_vol_20d", "US30Y_Rate_ret_20d"], "is_new": true}, {"model_id": "new_h3_CALM_LogisticRegression_N5_t1", "algo": "LogisticRegression", "regime": "CALM", "horizon": 3, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "HangSeng_HK_ret_5d", "EWM_Malaysia_zscore_60d", "SO_SouthernCo_ret_5d"], "is_new": true}, {"model_id": "new_h3_CALM_LogisticRegression_N5_t2", "algo": "LogisticRegression", "regime": "CALM", "horizon": 3, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "US3M_Rate_zscore_60d", "SBUX_vol_20d", "IYM_BasicMaterials_ret_20d"], "is_new": true}, {"model_id": "new_h3_CALM_LogisticRegression_N5_t3", "algo": "LogisticRegression", "regime": "CALM", "horizon": 3, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "US1Y_Rate_ret_5d", "DAX_Germany_vol_20d", "ASX_Australia_ret_5d"], "is_new": true}, {"model_id": "new_h3_CALM_LogisticRegression_N5_t4", "algo": "LogisticRegression", "regime": "CALM", "horizon": 3, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "ENB_EnbridgeInc_ret_1d", "WTI_Oil_FRED_zscore_60d", "CMCSA_ret_1d"], "is_new": true}, {"model_id": "new_h3_CALM_LogisticRegression_N5_t5", "algo": "LogisticRegression", "regime": "CALM", "horizon": 3, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EFFR_vol_20d", "MSTR_Bitcoin3_ret_5d", "LOW_Lowes_ret_20d"], "is_new": true}, {"model_id": "new_h3_CALM_LogisticRegression_N5_t6", "algo": "LogisticRegression", "regime": "CALM", "horizon": 3, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "3M_vol_20d", "EWL_Switzerland_vol_20d", "BDX_Becton_Dickinson_ret_20d"], "is_new": true}, {"model_id": "new_h3_CALM_LogisticRegression_N5_t7", "algo": "LogisticRegression", "regime": "CALM", "horizon": 3, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "Brent_Oil_FRED_ret_20d", "SBUX_zscore_60d", "TED_Spread_zscore_60d"], "is_new": true}, {"model_id": "new_h3_CALM_LogisticRegression_N8_t0", "algo": "LogisticRegression", "regime": "CALM", "horizon": 3, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EQIX_Equinix_ret_5d", "EWS_Singapore_ret_5d", "DHR_ret_1d", "SCHW_Schwab_ret_5d", "INTC_ret_1d", "TED_Spread_vol_20d"], "is_new": true}, {"model_id": "new_h3_CALM_LogisticRegression_N8_t1", "algo": "LogisticRegression", "regime": "CALM", "horizon": 3, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "XLK_Tech_zscore_60d", "ENB_EnbridgeInc_ret_1d", "AMZN_ret_5d", "NOC_Northrop_ret_20d", "EWL_Switzerland_vol_20d", "EXC_Exelon_zscore_60d"], "is_new": true}, {"model_id": "new_h3_CALM_LogisticRegression_N8_t2", "algo": "LogisticRegression", "regime": "CALM", "horizon": 3, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "PCAR_PaccarInc_ret_5d", "ENB_EnbridgeInc_ret_1d", "MS_MorganStanley_zscore_60d", "CLX_Clorox_vol_20d", "US1Y_Rate_ret_20d", "US3M_Rate_zscore_60d"], "is_new": true}, {"model_id": "new_h3_CALM_LogisticRegression_N8_t3", "algo": "LogisticRegression", "regime": "CALM", "horizon": 3, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "CPB_CampbellSoup_ret_20d", "AVB_AvalonBay_zscore_60d", "AMGN_Amgen_ret_1d", "BTI_BritishAmerican_ret_5d", "PFE_ret_1d", "XLB_Materials_zscore_60d"], "is_new": true}, {"model_id": "new_h3_CALM_LogisticRegression_N8_t4", "algo": "LogisticRegression", "regime": "CALM", "horizon": 3, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWM_Malaysia_ret_1d", "MS_MorganStanley_zscore_60d", "heston_var_ev_h3", "EOG_EOGResources_ret_5d", "EFFR_ret_1d", "EWA_Australia_zscore_60d"], "is_new": true}, {"model_id": "new_h3_CALM_LogisticRegression_N8_t5", "algo": "LogisticRegression", "regime": "CALM", "horizon": 3, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "spx_vol_5d", "PAYX_Paychex_vol_20d", "ASX_Australia_ret_5d", "EQIX_Equinix_ret_5d", "AORD_AUS_zscore_60d", "US3M_Rate_vol_20d"], "is_new": true}, {"model_id": "new_h3_CALM_LogisticRegression_N8_t6", "algo": "LogisticRegression", "regime": "CALM", "horizon": 3, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "Brent_Oil_FRED_ret_5d", "GILD_Gilead_ret_20d", "MS_MorganStanley_ret_1d", "XLV_Health_zscore_60d", "CI_Cigna_vol_20d", "EWQ_France_ret_20d"], "is_new": true}, {"model_id": "new_h3_CALM_LogisticRegression_N8_t7", "algo": "LogisticRegression", "regime": "CALM", "horizon": 3, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "DHR_vol_20d", "SBUX_zscore_60d", "EFFR_vol_20d", "ASX_Australia_vol_20d", "gjr_condvar_h1", "HD_ret_20d"], "is_new": true}, {"model_id": "new_h3_CALM_LogisticRegression_N10_t0", "algo": "LogisticRegression", "regime": "CALM", "horizon": 3, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWG_Germany_vol_20d", "LMT_LockheedMartin_ret_1d", "EWM_Malaysia_vol_20d", "EWQ_France_zscore_60d", "LOW_Lowes_ret_5d", "AMZN_ret_5d", "CTAS_Cintas_vol_20d", "M_Macys_vol_20d"], "is_new": true}, {"model_id": "new_h3_CALM_LogisticRegression_N10_t1", "algo": "LogisticRegression", "regime": "CALM", "horizon": 3, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "IWM_SmallCap_vol_20d", "BTI_BritishAmerican_ret_5d", "ASX_Australia_vol_20d", "IBEX_Spain_ret_20d", "US5Y_Rate_ret_5d", "GE_ret_1d", "MSTR_Bitcoin3_ret_1d", "Core_PCE_zscore_60d"], "is_new": true}, {"model_id": "new_h3_CALM_LogisticRegression_N10_t2", "algo": "LogisticRegression", "regime": "CALM", "horizon": 3, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "vix_acceleration_1d", "Brent_Oil_FRED_ret_5d", "Industrial_Production_zscore_60d", "CPB_CampbellSoup_ret_5d", "EWG_Germany_vol_20d", "EWL_Switzerland_zscore_60d", "spx_momentum_3d", "DAX_Germany_zscore_60d"], "is_new": true}, {"model_id": "new_h3_CALM_LogisticRegression_N10_t3", "algo": "LogisticRegression", "regime": "CALM", "horizon": 3, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "BDX_Becton_Dickinson_ret_20d", "VOD_Vodafone_zscore_60d", "DOW_Price_zscore_60d", "HUM_Humana_ret_5d", "3M_ret_5d", "Retail_Sales_zscore_60d", "IBEX_Spain_ret_20d", "CMCSA_ret_1d"], "is_new": true}, {"model_id": "new_h3_CALM_LogisticRegression_N10_t4", "algo": "LogisticRegression", "regime": "CALM", "horizon": 3, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "NOC_Northrop_ret_20d", "Industrial_Production_zscore_60d", "EWQ_France_zscore_60d", "EWQ_France_ret_20d", "US3M_Rate_vol_20d", "AXP_Amex_vol_20d", "EWL_Switzerland_zscore_60d", "MSTR_Bitcoin3_ret_5d"], "is_new": true}, {"model_id": "new_h3_CALM_LogisticRegression_N10_t5", "algo": "LogisticRegression", "regime": "CALM", "horizon": 3, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWG_Germany_ret_20d", "EWL_Switzerland_zscore_60d", "XLB_Materials_zscore_60d", "CPB_CampbellSoup_ret_5d", "VOD_Vodafone_zscore_60d", "INTC_ret_5d", "SLB_Schlumberger_ret_1d", "AMGN_Amgen_ret_1d"], "is_new": true}, {"model_id": "new_h3_CALM_LogisticRegression_N10_t6", "algo": "LogisticRegression", "regime": "CALM", "horizon": 3, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "CPB_CampbellSoup_vol_20d", "MS_MorganStanley_ret_5d", "DHR_vol_20d", "MSTR_Bitcoin3_ret_1d", "SBUX_vol_20d", "LOW_Lowes_ret_5d", "AXP_Amex_ret_20d", "CMCSA_ret_1d"], "is_new": true}, {"model_id": "new_h3_CALM_LogisticRegression_N10_t7", "algo": "LogisticRegression", "regime": "CALM", "horizon": 3, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "heston_var_ev_h7", "DAX_Germany_vol_20d", "GD_GeneralDynamics_zscore_60d", "MS_MorganStanley_ret_5d", "LOW_Lowes_ret_5d", "3M_vol_20d", "AMGN_Amgen_ret_1d", "EWM_Malaysia_ret_1d"], "is_new": true}, {"model_id": "new_h3_CALM_LogisticRegression_N12_t0", "algo": "LogisticRegression", "regime": "CALM", "horizon": 3, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "SLB_Schlumberger_ret_5d", "HD_zscore_60d", "MS_MorganStanley_ret_1d", "HD_ret_5d", "heston_var_ev_h5", "EMR_Emerson_ret_20d", "BLK_BlackRock_zscore_60d", "CPB_CampbellSoup_ret_5d", "SJM_JM_Smucker_ret_5d", "CPB_CampbellSoup_zscore_60d"], "is_new": true}, {"model_id": "new_h3_CALM_LogisticRegression_N12_t1", "algo": "LogisticRegression", "regime": "CALM", "horizon": 3, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EOG_EOGResources_ret_5d", "3M_ret_5d", "CPB_CampbellSoup_ret_20d", "MSTR_Bitcoin3_ret_1d", "LMT_LockheedMartin_ret_1d", "CPB_CampbellSoup_vol_20d", "WTI_Oil_FRED_zscore_60d", "T10Y2Y_Spread_ret_5d", "T_ret_1d", "XLB_Materials_zscore_60d"], "is_new": true}, {"model_id": "new_h3_CALM_LogisticRegression_N12_t2", "algo": "LogisticRegression", "regime": "CALM", "horizon": 3, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "SBUX_vol_20d", "LMT_LockheedMartin_ret_1d", "ASX_Australia_ret_5d", "PCAR_PaccarInc_ret_5d", "JNJ_ret_1d", "Core_PCE_zscore_60d", "US6M_Rate_ret_20d", "EMR_Emerson_ret_20d", "MS_MorganStanley_ret_1d", "PG_ret_20d"], "is_new": true}, {"model_id": "new_h3_CALM_LogisticRegression_N12_t3", "algo": "LogisticRegression", "regime": "CALM", "horizon": 3, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "M_Macys_vol_20d", "CPB_CampbellSoup_ret_20d", "MS_MorganStanley_ret_5d", "heston_var_ev_h7", "EFFR_vol_20d", "spx_vol_5d", "DIS_vol_20d", "heston_var_ev_h3", "SO_SouthernCo_ret_5d", "XOM_ret_20d"], "is_new": true}, {"model_id": "new_h3_CALM_LogisticRegression_N12_t4", "algo": "LogisticRegression", "regime": "CALM", "horizon": 3, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "NWL_Newell_ret_20d", "TM_Telephone_ret_1d", "heston_ev_h3", "US1Y_Rate_ret_5d", "heston_var_ev_h3", "SJM_JM_Smucker_ret_1d", "ENB_EnbridgeInc_ret_1d", "AMD_ret_1d", "DHR_vol_20d", "3M_ret_5d"], "is_new": true}, {"model_id": "new_h3_CALM_LogisticRegression_N12_t5", "algo": "LogisticRegression", "regime": "CALM", "horizon": 3, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWM_Malaysia_vol_20d", "MSTR_Bitcoin3_ret_5d", "Industrial_Production_zscore_60d", "HD_ret_20d", "SBUX_vol_20d", "MSTR_Bitcoin3_ret_1d", "HUM_Humana_ret_5d", "CLX_Clorox_vol_20d", "EWY_Korea_zscore_60d", "HD_zscore_60d"], "is_new": true}, {"model_id": "new_h3_CALM_LogisticRegression_N12_t6", "algo": "LogisticRegression", "regime": "CALM", "horizon": 3, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "HUM_Humana_ret_5d", "Industrial_Production_zscore_60d", "EWL_Switzerland_vol_20d", "BTI_BritishAmerican_ret_20d", "LUV_SouthwestAir_ret_5d", "US6M_Rate_ret_20d", "MO_AltriaMG_ret_1d", "LMT_LockheedMartin_vol_20d", "AMD_ret_5d", "US30Y_Rate_ret_20d"], "is_new": true}, {"model_id": "new_h3_CALM_LogisticRegression_N12_t7", "algo": "LogisticRegression", "regime": "CALM", "horizon": 3, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "Nikkei_Japan_vol_20d", "US7Y_Rate_ret_20d", "LMT_LockheedMartin_ret_1d", "HangSeng_HK_vol_20d", "PPL_PPL_ret_1d", "EWC_Canada_zscore_60d", "INTC_ret_1d", "NWL_Newell_ret_20d", "EXC_Exelon_ret_1d", "DE_Deere_vol_20d"], "is_new": true}, {"model_id": "new_h3_CALM_LogisticRegression_N15_t0", "algo": "LogisticRegression", "regime": "CALM", "horizon": 3, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "NFCI_ret_5d", "gjr_condvar_h1", "CMCSA_ret_1d", "ASX_Australia_vol_20d", "XLK_Tech_zscore_60d", "AMZN_ret_5d", "US3M_Rate_vol_20d", "T10Y2Y_Spread_ret_5d", "EFFR_vol_20d", "EXC_Exelon_zscore_60d", "ITT_ITTInc_ret_5d", "BDX_Becton_Dickinson_ret_20d", "heston_var_ev_h7"], "is_new": true}, {"model_id": "new_h3_CALM_LogisticRegression_N15_t1", "algo": "LogisticRegression", "regime": "CALM", "horizon": 3, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "vix_acceleration_1d", "HangSeng_HK_ret_1d", "PAYX_Paychex_ret_20d", "AVB_AvalonBay_zscore_60d", "DOW_Price_zscore_60d", "EWC_Canada_zscore_60d", "NVDA_vol_20d", "3M_vol_20d", "ORCL_zscore_60d", "HUM_Humana_ret_5d", "CTAS_Cintas_vol_20d", "SBUX_zscore_60d", "BLK_BlackRock_zscore_60d"], "is_new": true}, {"model_id": "new_h3_CALM_LogisticRegression_N15_t2", "algo": "LogisticRegression", "regime": "CALM", "horizon": 3, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "LOW_Lowes_ret_5d", "EOG_EOGResources_ret_5d", "IYR_US_REIT2_zscore_60d", "ITT_ITTInc_ret_5d", "EWG_Germany_ret_20d", "spx_vol_5d", "spx_momentum_3d", "CCI_CrownCastle_vol_20d", "EWA_Australia_ret_1d", "EXC_Exelon_ret_1d", "CMCSA_ret_1d", "PAYX_Paychex_zscore_60d", "NVDA_vol_20d"], "is_new": true}, {"model_id": "new_h3_CALM_LogisticRegression_N15_t3", "algo": "LogisticRegression", "regime": "CALM", "horizon": 3, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "HangSeng_HK_ret_1d", "BTI_BritishAmerican_ret_5d", "SBUX_zscore_60d", "MO_AltriaMG_ret_1d", "GD_GeneralDynamics_zscore_60d", "ASX_Australia_ret_5d", "CPB_CampbellSoup_ret_20d", "MS_MorganStanley_ret_1d", "EWY_Korea_ret_20d", "PAYX_Paychex_vol_20d", "XLV_Health_zscore_60d", "ASX_Australia_vol_20d", "LMT_LockheedMartin_ret_1d"], "is_new": true}, {"model_id": "new_h3_CALM_LogisticRegression_N15_t4", "algo": "LogisticRegression", "regime": "CALM", "horizon": 3, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "PAYX_Paychex_vol_20d", "EWA_Australia_ret_1d", "XLV_Health_zscore_60d", "EWG_Germany_vol_20d", "EWJ_Japan_vol_20d", "XLK_Tech_zscore_60d", "Michigan_Sentiment_ret_20d", "XLY_Disc_vol_20d", "gjr_condvar_h1", "CPB_CampbellSoup_vol_20d", "AMZN_ret_5d", "TXN_vol_20d", "EMR_Emerson_ret_20d"], "is_new": true}, {"model_id": "new_h3_CALM_LogisticRegression_N15_t5", "algo": "LogisticRegression", "regime": "CALM", "horizon": 3, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "CPB_CampbellSoup_zscore_60d", "DE_Deere_vol_20d", "BLK_BlackRock_zscore_60d", "MSTR_Bitcoin3_ret_20d", "HD_ret_20d", "MSTR_Bitcoin3_ret_5d", "HD_ret_1d", "NEE_NextEra_ret_20d", "Retail_Sales_zscore_60d", "MSTR_Bitcoin3_ret_1d", "EXC_Exelon_zscore_60d", "PG_ret_20d", "US1Y_Rate_ret_5d"], "is_new": true}, {"model_id": "new_h3_CALM_LogisticRegression_N15_t6", "algo": "LogisticRegression", "regime": "CALM", "horizon": 3, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "AXP_Amex_vol_20d", "US1Y_Rate_ret_5d", "EXC_Exelon_zscore_60d", "LMT_LockheedMartin_vol_20d", "BDX_Becton_Dickinson_ret_20d", "Brent_Oil_FRED_ret_5d", "hmm_p_stress", "HangSeng_HK_vol_20d", "EWY_Korea_ret_20d", "US3Y_Rate_ret_5d", "US6M_Rate_ret_20d", "US3M_Rate_vol_20d", "EFFR_ret_1d"], "is_new": true}, {"model_id": "new_h3_CALM_LogisticRegression_N15_t7", "algo": "LogisticRegression", "regime": "CALM", "horizon": 3, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "SO_SouthernCo_ret_5d", "Nikkei_Japan_zscore_60d", "Core_PCE_zscore_60d", "AMD_ret_1d", "DHR_vol_20d", "T10Y2Y_Spread_ret_5d", "LLY_zscore_60d", "XOM_ret_20d", "IYR_US_REIT2_zscore_60d", "CPB_CampbellSoup_zscore_60d", "EWY_Korea_zscore_60d", "EWQ_France_zscore_60d", "TED_Spread_zscore_60d"], "is_new": true}, {"model_id": "new_h3_CALM_LogisticRegression_N20_t0", "algo": "LogisticRegression", "regime": "CALM", "horizon": 3, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "DHR_vol_20d", "AMD_ret_1d", "EWY_Korea_ret_20d", "ES_Evergy_ret_1d", "TM_Telephone_vol_20d", "NEE_NextEra_ret_20d", "Industrial_Production_zscore_60d", "US3M_Rate_vol_20d", "Core_PCE_zscore_60d", "TM_Telephone_ret_1d", "DAX_Germany_vol_20d", "TED_Spread_zscore_60d", "ASX_Australia_vol_20d", "LMT_LockheedMartin_ret_1d", "EWG_Germany_vol_20d", "XLB_Materials_zscore_60d", "AVB_AvalonBay_zscore_60d", "EWA_Australia_zscore_60d"], "is_new": true}, {"model_id": "new_h3_CALM_LogisticRegression_N20_t1", "algo": "LogisticRegression", "regime": "CALM", "horizon": 3, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "AMD_ret_5d", "vix_mean_abs_ret_5d", "TM_Telephone_vol_20d", "EWM_Malaysia_ret_1d", "BA_ret_1d", "EWM_Malaysia_vol_20d", "US30Y_Rate_ret_20d", "TED_Spread_vol_20d", "HD_zscore_60d", "AMGN_Amgen_ret_1d", "CPB_CampbellSoup_vol_20d", "Nikkei_Japan_zscore_60d", "heston_var_ev_h3", "QQQ_vol_20d", "PG_ret_20d", "M_Macys_vol_20d", "VRP_ma5", "DHR_vol_20d"], "is_new": true}, {"model_id": "new_h3_CALM_LogisticRegression_N20_t2", "algo": "LogisticRegression", "regime": "CALM", "horizon": 3, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "IYM_BasicMaterials_ret_20d", "AMD_ret_5d", "SBUX_zscore_60d", "CPB_CampbellSoup_ret_5d", "US6M_Rate_ret_20d", "PFE_ret_1d", "ASX_Australia_ret_5d", "XLB_Materials_zscore_60d", "LMT_LockheedMartin_vol_20d", "SBUX_vol_20d", "HUM_Humana_ret_5d", "HangSeng_HK_vol_20d", "LMT_LockheedMartin_ret_1d", "HD_ret_1d", "AMZN_ret_5d", "ORCL_vol_20d", "VRP_ma5", "Core_CPI_zscore_60d"], "is_new": true}, {"model_id": "new_h3_CALM_LogisticRegression_N20_t3", "algo": "LogisticRegression", "regime": "CALM", "horizon": 3, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "PPL_PPL_ret_1d", "EWH_HongKong_ret_5d", "US1Y_Rate_ret_20d", "heston_var_ev_h5", "HangSeng_HK_vol_20d", "EWS_Singapore_ret_5d", "Core_CPI_zscore_60d", "heston_ev_h3", "INTC_ret_5d", "MRK_Merck_zscore_60d", "NFCI_ret_5d", "SCHW_Schwab_ret_5d", "NWL_Newell_ret_20d", "XLF_Fin_vol_20d", "INTC_ret_1d", "WTI_Oil_FRED_zscore_60d", "PG_ret_20d", "XOM_ret_1d"], "is_new": true}, {"model_id": "new_h3_CALM_LogisticRegression_N20_t4", "algo": "LogisticRegression", "regime": "CALM", "horizon": 3, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWM_Malaysia_vol_20d", "ENB_EnbridgeInc_ret_1d", "QQQ_vol_20d", "EWL_Switzerland_vol_20d", "FedFunds_zscore_60d", "SBUX_zscore_60d", "AVB_AvalonBay_zscore_60d", "EWA_Australia_ret_1d", "heston_ev_h3", "XLV_Health_zscore_60d", "CCI_CrownCastle_vol_20d", "US3Y_Rate_ret_5d", "vix_acceleration_1d", "IWM_SmallCap_vol_20d", "NEE_NextEra_ret_20d", "Michigan_Sentiment_ret_20d", "ITT_ITTInc_ret_5d", "CPB_CampbellSoup_zscore_60d"], "is_new": true}, {"model_id": "new_h3_CALM_LogisticRegression_N20_t5", "algo": "LogisticRegression", "regime": "CALM", "horizon": 3, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWA_Australia_ret_1d", "AMT_AmericanTower_ret_1d", "PAYX_Paychex_vol_20d", "MS_MorganStanley_zscore_60d", "SPY_zscore_60d", "hmm_p_stress", "EWM_Malaysia_zscore_60d", "DOW_Price_zscore_60d", "US5Y_Rate_ret_5d", "SLB_Schlumberger_ret_1d", "PAYX_Paychex_ret_20d", "MSTR_Bitcoin3_ret_5d", "T_ret_1d", "CI_Cigna_vol_20d", "heston_var_ev_h5", "heston_ev_h3", "Nikkei_Japan_zscore_60d", "ASX_Australia_ret_5d"], "is_new": true}, {"model_id": "new_h3_CALM_LogisticRegression_N20_t6", "algo": "LogisticRegression", "regime": "CALM", "horizon": 3, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "3M_vol_20d", "EFFR_vol_20d", "GD_GeneralDynamics_zscore_60d", "ES_Evergy_ret_1d", "EWQ_France_ret_20d", "Nikkei_Japan_vol_20d", "EWQ_France_zscore_60d", "PAYX_Paychex_ret_20d", "EWY_Korea_zscore_60d", "spx_vol_5d", "EWL_Switzerland_vol_20d", "PG_ret_20d", "EWY_Korea_ret_20d", "TGT_Target_zscore_60d", "AXP_Amex_ret_20d", "BTI_BritishAmerican_ret_5d", "Brent_Oil_FRED_ret_5d", "TED_Spread_vol_20d"], "is_new": true}, {"model_id": "new_h3_CALM_LogisticRegression_N20_t7", "algo": "LogisticRegression", "regime": "CALM", "horizon": 3, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWA_Australia_ret_1d", "LLY_zscore_60d", "SBUX_ret_5d", "Core_PCE_zscore_60d", "ORCL_vol_20d", "US30Y_Rate_ret_20d", "DHR_vol_20d", "NFCI_ret_5d", "DAX_Germany_vol_20d", "BTI_BritishAmerican_ret_20d", "CPB_CampbellSoup_zscore_60d", "HD_ret_5d", "PG_ret_20d", "MSTR_Bitcoin3_ret_20d", "EOG_EOGResources_ret_5d", "AMZN_ret_5d", "PAYX_Paychex_ret_20d", "EWG_Germany_vol_20d"], "is_new": true}, {"model_id": "new_h3_CALM_LogisticRegression_N25_t0", "algo": "LogisticRegression", "regime": "CALM", "horizon": 3, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "LLY_zscore_60d", "IBEX_Spain_ret_20d", "Core_PCE_zscore_60d", "HD_ret_1d", "ITT_ITTInc_ret_5d", "LMT_LockheedMartin_ret_1d", "NOC_Northrop_ret_20d", "HangSeng_HK_vol_20d", "PAYX_Paychex_zscore_60d", "EWA_Australia_zscore_60d", "BTI_BritishAmerican_ret_20d", "CPB_CampbellSoup_ret_5d", "MS_MorganStanley_zscore_60d", "SO_SouthernCo_ret_5d", "TGT_Target_zscore_60d", "IWM_SmallCap_vol_20d", "VRP_ma5", "EQIX_Equinix_ret_5d", "Michigan_Sentiment_ret_20d", "WTI_Oil_FRED_zscore_60d", "EWM_Malaysia_zscore_60d", "QQQ_vol_20d", "AMGN_Amgen_ret_1d"], "is_new": true}, {"model_id": "new_h3_CALM_LogisticRegression_N25_t1", "algo": "LogisticRegression", "regime": "CALM", "horizon": 3, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "AXP_Amex_ret_20d", "DAX_Germany_vol_20d", "HangSeng_HK_ret_1d", "T_ret_1d", "LUV_SouthwestAir_ret_5d", "QQQ_vol_20d", "SLB_Schlumberger_ret_1d", "CPB_CampbellSoup_vol_20d", "Michigan_Sentiment_ret_20d", "MRK_Merck_zscore_60d", "Nikkei_Japan_zscore_60d", "AMD_ret_1d", "SPY_zscore_60d", "TED_Spread_zscore_60d", "AXP_Amex_vol_20d", "FedFunds_zscore_60d", "BTI_BritishAmerican_ret_5d", "SLB_Schlumberger_ret_5d", "ASX_Australia_vol_20d", "DE_Deere_ret_5d", "HD_ret_20d", "VRP_ma5", "M_Macys_vol_20d"], "is_new": true}, {"model_id": "new_h3_CALM_LogisticRegression_N25_t2", "algo": "LogisticRegression", "regime": "CALM", "horizon": 3, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "BTI_BritishAmerican_ret_20d", "PLD_Prologis_ret_5d", "EWG_Germany_vol_20d", "FedFunds_zscore_60d", "MSTR_Bitcoin3_ret_1d", "ASX_Australia_ret_5d", "TXN_vol_20d", "PFE_ret_1d", "EWL_Switzerland_vol_20d", "ORCL_vol_20d", "EWG_Germany_ret_20d", "CCI_CrownCastle_vol_20d", "spx_momentum_3d", "US3M_Rate_vol_20d", "EWA_Australia_zscore_60d", "CMCSA_ret_1d", "SPY_zscore_60d", "CI_Cigna_vol_20d", "EWA_Australia_ret_1d", "PAYX_Paychex_zscore_60d", "DE_Deere_vol_20d", "DOW_Price_zscore_60d", "gjr_condvar_h1"], "is_new": true}, {"model_id": "new_h3_CALM_LogisticRegression_N25_t3", "algo": "LogisticRegression", "regime": "CALM", "horizon": 3, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "3M_vol_20d", "hmm_p_stress", "EWM_Malaysia_vol_20d", "3M_ret_5d", "heston_var_ev_h5", "TED_Spread_vol_20d", "spx_vol_5d", "GILD_Gilead_ret_20d", "US3M_Rate_zscore_60d", "CTAS_Cintas_vol_20d", "CCI_CrownCastle_vol_20d", "M_Macys_vol_20d", "HD_ret_5d", "SBUX_zscore_60d", "NFCI_ret_5d", "EXC_Exelon_ret_1d", "US6M_Rate_ret_20d", "ES_Evergy_ret_1d", "Retail_Sales_zscore_60d", "EWC_Canada_zscore_60d", "US30Y_Rate_ret_20d", "NEE_NextEra_ret_20d", "US1Y_Rate_ret_5d"], "is_new": true}, {"model_id": "new_h3_CALM_LogisticRegression_N25_t4", "algo": "LogisticRegression", "regime": "CALM", "horizon": 3, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "hmm_p_stress", "DE_Deere_ret_5d", "EMR_Emerson_ret_20d", "heston_ev_h3", "SBUX_vol_20d", "US1Y_Rate_ret_5d", "CPB_CampbellSoup_vol_20d", "vix_mean_abs_ret_5d", "Brent_Oil_FRED_ret_5d", "LUV_SouthwestAir_ret_5d", "LLY_zscore_60d", "EWH_HongKong_ret_5d", "US6M_Rate_ret_20d", "ES_Evergy_ret_1d", "spx_momentum_3d", "SJM_JM_Smucker_ret_5d", "TGT_Target_zscore_60d", "EWY_Korea_ret_20d", "ASX_Australia_ret_5d", "EOG_EOGResources_vol_20d", "CLX_Clorox_vol_20d", "EWG_Germany_ret_20d", "MS_MorganStanley_zscore_60d"], "is_new": true}, {"model_id": "new_h3_CALM_LogisticRegression_N25_t5", "algo": "LogisticRegression", "regime": "CALM", "horizon": 3, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "hmm_p_stress", "EWQ_France_zscore_60d", "EWS_Singapore_ret_5d", "SJM_JM_Smucker_ret_5d", "IBEX_Spain_ret_20d", "Brent_Oil_FRED_ret_20d", "QQQ_vol_20d", "T10Y2Y_Spread_ret_5d", "PCAR_PaccarInc_ret_5d", "Michigan_Sentiment_ret_20d", "AMD_ret_5d", "EFFR_ret_1d", "SLB_Schlumberger_ret_5d", "SCHW_Schwab_ret_5d", "spx_abs_ret_max_5d", "WTI_Oil_FRED_zscore_60d", "gjr_condvar_h1", "PPL_PPL_ret_1d", "EFFR_vol_20d", "NFCI_ret_5d", "EWY_Korea_zscore_60d", "TED_Spread_zscore_60d", "XOM_ret_1d"], "is_new": true}, {"model_id": "new_h3_CALM_LogisticRegression_N25_t6", "algo": "LogisticRegression", "regime": "CALM", "horizon": 3, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWY_Korea_ret_20d", "VOD_Vodafone_zscore_60d", "CLX_Clorox_vol_20d", "NOC_Northrop_ret_20d", "SBUX_ret_5d", "TM_Telephone_ret_1d", "Brent_Oil_FRED_ret_5d", "EOG_EOGResources_vol_20d", "TED_Spread_zscore_60d", "SLB_Schlumberger_ret_5d", "EWS_Singapore_ret_5d", "PAYX_Paychex_ret_20d", "XLK_Tech_zscore_60d", "EWL_Switzerland_zscore_60d", "XLB_Materials_zscore_60d", "TM_Telephone_vol_20d", "IYM_BasicMaterials_ret_20d", "US3M_Rate_zscore_60d", "EWM_Malaysia_ret_1d", "BDX_Becton_Dickinson_ret_20d", "US1Y_Rate_ret_5d", "EFFR_ret_1d", "ES_Evergy_ret_1d"], "is_new": true}, {"model_id": "new_h3_CALM_LogisticRegression_N25_t7", "algo": "LogisticRegression", "regime": "CALM", "horizon": 3, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "US3Y_Rate_ret_5d", "LMT_LockheedMartin_ret_1d", "EWY_Korea_ret_20d", "VRP_ma5", "INTC_ret_5d", "EWS_Singapore_ret_5d", "NVDA_vol_20d", "spx_momentum_3d", "SPY_zscore_60d", "DOW_Price_zscore_60d", "hmm_p_stress", "EOG_EOGResources_vol_20d", "AVB_AvalonBay_zscore_60d", "ORCL_zscore_60d", "ENB_EnbridgeInc_ret_1d", "HangSeng_HK_ret_1d", "SLB_Schlumberger_ret_5d", "NOC_Northrop_ret_20d", "CPB_CampbellSoup_ret_5d", "CI_Cigna_vol_20d", "JNJ_ret_1d", "Nikkei_Japan_zscore_60d", "EFFR_vol_20d"], "is_new": true}, {"model_id": "new_h3_CALM_LogisticRegression_N30_t0", "algo": "LogisticRegression", "regime": "CALM", "horizon": 3, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "US1Y_Rate_ret_20d", "AORD_AUS_zscore_60d", "BTI_BritishAmerican_ret_20d", "US5Y_Rate_ret_5d", "HD_ret_1d", "MSTR_Bitcoin3_ret_5d", "SBUX_zscore_60d", "ASX_Australia_vol_20d", "CI_Cigna_vol_20d", "GILD_Gilead_ret_20d", "Brent_Oil_FRED_ret_5d", "DOW_Price_zscore_60d", "US30Y_Rate_ret_20d", "3M_ret_5d", "EXC_Exelon_ret_1d", "HUM_Humana_ret_5d", "ORCL_zscore_60d", "CPB_CampbellSoup_vol_20d", "EFFR_vol_20d", "heston_var_ev_h5", "XOM_ret_1d", "EWG_Germany_ret_20d", "NVDA_vol_20d", "DAX_Germany_zscore_60d", "HD_ret_20d", "T_ret_1d", "PAYX_Paychex_vol_20d", "HangSeng_HK_vol_20d"], "is_new": true}, {"model_id": "new_h3_CALM_LogisticRegression_N30_t1", "algo": "LogisticRegression", "regime": "CALM", "horizon": 3, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWJ_Japan_vol_20d", "PAYX_Paychex_vol_20d", "TXN_vol_20d", "heston_var_ev_h7", "US1Y_Rate_ret_5d", "NWL_Newell_ret_20d", "BTI_BritishAmerican_ret_5d", "Retail_Sales_zscore_60d", "EWL_Switzerland_zscore_60d", "DE_Deere_ret_5d", "PAYX_Paychex_zscore_60d", "PFE_ret_1d", "NEE_NextEra_ret_20d", "AMZN_ret_5d", "NOC_Northrop_ret_20d", "EWA_Australia_ret_1d", "IYM_BasicMaterials_ret_20d", "BTI_BritishAmerican_ret_20d", "EWQ_France_zscore_60d", "XOM_ret_1d", "spx_abs_ret_max_5d", "US3M_Rate_zscore_60d", "US7Y_Rate_ret_20d", "AXP_Amex_vol_20d", "Nikkei_Japan_vol_20d", "LOW_Lowes_ret_5d", "QQQ_vol_20d", "HD_zscore_60d"], "is_new": true}, {"model_id": "new_h3_CALM_LogisticRegression_N30_t2", "algo": "LogisticRegression", "regime": "CALM", "horizon": 3, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "NOC_Northrop_ret_20d", "Michigan_Sentiment_ret_20d", "EWS_Singapore_ret_5d", "EWY_Korea_ret_20d", "QQQ_vol_20d", "MRK_Merck_zscore_60d", "vix_mean_abs_ret_5d", "CI_Cigna_vol_20d", "SLB_Schlumberger_ret_1d", "GE_ret_1d", "LMT_LockheedMartin_vol_20d", "EWL_Switzerland_zscore_60d", "JNJ_ret_1d", "SBUX_ret_5d", "PAYX_Paychex_zscore_60d", "XLY_Disc_vol_20d", "SCHW_Schwab_ret_5d", "TED_Spread_vol_20d", "LOW_Lowes_ret_5d", "US3M_Rate_vol_20d", "DHR_ret_1d", "GILD_Gilead_ret_20d", "Industrial_Production_zscore_60d", "heston_var_ev_h3", "EWM_Malaysia_zscore_60d", "NFCI_ret_5d", "Brent_Oil_FRED_ret_20d", "TM_Telephone_ret_1d"], "is_new": true}, {"model_id": "new_h3_CALM_LogisticRegression_N30_t3", "algo": "LogisticRegression", "regime": "CALM", "horizon": 3, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "IBEX_Spain_ret_20d", "TM_Telephone_vol_20d", "US3M_Rate_vol_20d", "ENB_EnbridgeInc_ret_1d", "EWA_Australia_ret_1d", "XLY_Disc_vol_20d", "MSTR_Bitcoin3_ret_1d", "ASX_Australia_vol_20d", "HD_ret_20d", "DE_Deere_ret_5d", "GD_GeneralDynamics_zscore_60d", "gjr_condvar_h1", "IWM_SmallCap_vol_20d", "3M_vol_20d", "EQR_Equity_ret_1d", "LMT_LockheedMartin_ret_1d", "MS_MorganStanley_ret_1d", "SJM_JM_Smucker_ret_1d", "BLK_BlackRock_zscore_60d", "EFFR_ret_1d", "Core_PCE_zscore_60d", "Retail_Sales_zscore_60d", "SPY_zscore_60d", "EOG_EOGResources_vol_20d", "IYR_US_REIT2_zscore_60d", "US1Y_Rate_ret_5d", "TXN_vol_20d", "Brent_Oil_FRED_ret_20d"], "is_new": true}, {"model_id": "new_h3_CALM_LogisticRegression_N30_t4", "algo": "LogisticRegression", "regime": "CALM", "horizon": 3, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "ORCL_zscore_60d", "NFCI_ret_5d", "EWH_HongKong_ret_5d", "EMR_Emerson_ret_20d", "VOD_Vodafone_zscore_60d", "HD_zscore_60d", "3M_ret_5d", "EWM_Malaysia_ret_1d", "SLB_Schlumberger_ret_5d", "AORD_AUS_zscore_60d", "HangSeng_HK_ret_5d", "EWJ_Japan_vol_20d", "SBUX_zscore_60d", "EXC_Exelon_ret_1d", "CMCSA_ret_1d", "EFFR_vol_20d", "ORCL_vol_20d", "T_ret_1d", "AMD_ret_1d", "SO_SouthernCo_ret_5d", "MSTR_Bitcoin3_ret_1d", "INTC_ret_5d", "spx_abs_ret_max_5d", "Core_PCE_zscore_60d", "AMD_ret_5d", "TED_Spread_vol_20d", "vix_acceleration_1d", "CPB_CampbellSoup_vol_20d"], "is_new": true}, {"model_id": "new_h3_CALM_LogisticRegression_N30_t5", "algo": "LogisticRegression", "regime": "CALM", "horizon": 3, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "SJM_JM_Smucker_ret_5d", "EWG_Germany_vol_20d", "XOM_ret_1d", "vix_acceleration_1d", "CCI_CrownCastle_vol_20d", "EWQ_France_ret_20d", "DAX_Germany_zscore_60d", "spx_vol_5d", "HangSeng_HK_ret_5d", "ASX_Australia_ret_5d", "ENB_EnbridgeInc_ret_1d", "PPL_PPL_ret_1d", "LUV_SouthwestAir_ret_5d", "VOD_Vodafone_zscore_60d", "HD_zscore_60d", "EQR_Equity_ret_1d", "LOW_Lowes_ret_20d", "AVB_AvalonBay_zscore_60d", "BTI_BritishAmerican_ret_20d", "Michigan_Sentiment_ret_20d", "MRK_Merck_zscore_60d", "PAYX_Paychex_vol_20d", "heston_var_ev_h5", "US7Y_Rate_ret_20d", "INTC_ret_5d", "QQQ_vol_20d", "heston_var_ev_h3", "T10Y2Y_Spread_ret_5d"], "is_new": true}, {"model_id": "new_h3_CALM_LogisticRegression_N30_t6", "algo": "LogisticRegression", "regime": "CALM", "horizon": 3, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "AMD_ret_1d", "vix_acceleration_1d", "SCHW_Schwab_ret_5d", "3M_ret_5d", "JNJ_ret_1d", "MS_MorganStanley_ret_5d", "BLK_BlackRock_zscore_60d", "Industrial_Production_zscore_60d", "INTC_ret_1d", "Retail_Sales_zscore_60d", "EWH_HongKong_ret_5d", "T10Y2Y_Spread_ret_5d", "HD_ret_20d", "Core_CPI_zscore_60d", "BA_ret_1d", "US3Y_Rate_ret_5d", "Michigan_Sentiment_ret_20d", "3M_vol_20d", "AORD_AUS_zscore_60d", "Nikkei_Japan_zscore_60d", "MSTR_Bitcoin3_ret_20d", "XLY_Disc_vol_20d", "NEE_NextEra_ret_20d", "BTI_BritishAmerican_ret_5d", "spx_abs_ret_max_5d", "EWA_Australia_zscore_60d", "CCI_CrownCastle_vol_20d", "ORCL_zscore_60d"], "is_new": true}, {"model_id": "new_h3_CALM_LogisticRegression_N30_t7", "algo": "LogisticRegression", "regime": "CALM", "horizon": 3, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "HUM_Humana_ret_5d", "HD_zscore_60d", "MRK_Merck_zscore_60d", "MSTR_Bitcoin3_ret_1d", "WTI_Oil_FRED_zscore_60d", "PG_ret_20d", "VRP_ma5", "CMCSA_ret_1d", "SLB_Schlumberger_ret_5d", "CI_Cigna_vol_20d", "vix_mean_abs_ret_5d", "EWJ_Japan_vol_20d", "EWM_Malaysia_vol_20d", "MS_MorganStanley_ret_1d", "PAYX_Paychex_zscore_60d", "IYM_BasicMaterials_ret_20d", "PFE_ret_1d", "IYR_US_REIT2_zscore_60d", "LOW_Lowes_ret_5d", "FedFunds_zscore_60d", "MS_MorganStanley_ret_5d", "AMGN_Amgen_ret_1d", "MSTR_Bitcoin3_ret_5d", "US1Y_Rate_ret_5d", "EWA_Australia_ret_1d", "GD_GeneralDynamics_zscore_60d", "EQR_Equity_ret_1d", "NOC_Northrop_ret_20d"], "is_new": true}, {"model_id": "new_h3_NORMAL_XGBoost_N5_t0", "algo": "XGBoost", "regime": "NORMAL", "horizon": 3, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "TED_Spread_vol_20d", "TXN_vol_20d", "EFFR_vol_20d"], "is_new": true}, {"model_id": "new_h3_NORMAL_XGBoost_N5_t1", "algo": "XGBoost", "regime": "NORMAL", "horizon": 3, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "MSTR_Bitcoin3_ret_1d", "AXP_Amex_vol_20d", "TXN_vol_20d"], "is_new": true}, {"model_id": "new_h3_NORMAL_XGBoost_N5_t2", "algo": "XGBoost", "regime": "NORMAL", "horizon": 3, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "SPY_zscore_60d", "US1Y_Rate_ret_5d", "ASX_Australia_vol_20d"], "is_new": true}, {"model_id": "new_h3_NORMAL_XGBoost_N5_t3", "algo": "XGBoost", "regime": "NORMAL", "horizon": 3, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWA_Australia_zscore_60d", "NVDA_vol_20d", "EWM_Malaysia_ret_1d"], "is_new": true}, {"model_id": "new_h3_NORMAL_XGBoost_N5_t4", "algo": "XGBoost", "regime": "NORMAL", "horizon": 3, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "CPB_CampbellSoup_vol_20d", "DHR_ret_1d", "MS_MorganStanley_ret_1d"], "is_new": true}, {"model_id": "new_h3_NORMAL_XGBoost_N5_t5", "algo": "XGBoost", "regime": "NORMAL", "horizon": 3, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "ENB_EnbridgeInc_ret_1d", "EFFR_vol_20d", "spx_abs_ret_max_5d"], "is_new": true}, {"model_id": "new_h3_NORMAL_XGBoost_N5_t6", "algo": "XGBoost", "regime": "NORMAL", "horizon": 3, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "HUM_Humana_ret_5d", "EWL_Switzerland_vol_20d", "BTI_BritishAmerican_ret_5d"], "is_new": true}, {"model_id": "new_h3_NORMAL_XGBoost_N5_t7", "algo": "XGBoost", "regime": "NORMAL", "horizon": 3, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "GILD_Gilead_ret_20d", "ITT_ITTInc_ret_5d", "IYM_BasicMaterials_ret_20d"], "is_new": true}, {"model_id": "new_h3_NORMAL_XGBoost_N8_t0", "algo": "XGBoost", "regime": "NORMAL", "horizon": 3, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "ES_Evergy_ret_1d", "JNJ_ret_1d", "M_Macys_vol_20d", "MS_MorganStanley_ret_1d", "US1Y_Rate_ret_5d", "SBUX_vol_20d"], "is_new": true}, {"model_id": "new_h3_NORMAL_XGBoost_N8_t1", "algo": "XGBoost", "regime": "NORMAL", "horizon": 3, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "heston_var_ev_h5", "EWH_HongKong_ret_5d", "NVDA_vol_20d", "LMT_LockheedMartin_ret_1d", "PAYX_Paychex_vol_20d", "TED_Spread_vol_20d"], "is_new": true}, {"model_id": "new_h3_NORMAL_XGBoost_N8_t2", "algo": "XGBoost", "regime": "NORMAL", "horizon": 3, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "CPB_CampbellSoup_zscore_60d", "BA_ret_1d", "HD_ret_20d", "CPB_CampbellSoup_ret_5d", "EWQ_France_ret_20d", "M_Macys_vol_20d"], "is_new": true}, {"model_id": "new_h3_NORMAL_XGBoost_N8_t3", "algo": "XGBoost", "regime": "NORMAL", "horizon": 3, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "SBUX_vol_20d", "XLF_Fin_vol_20d", "LOW_Lowes_ret_5d", "US6M_Rate_ret_20d", "spx_momentum_3d", "heston_var_ev_h5"], "is_new": true}, {"model_id": "new_h3_NORMAL_XGBoost_N8_t4", "algo": "XGBoost", "regime": "NORMAL", "horizon": 3, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "WTI_Oil_FRED_zscore_60d", "MSTR_Bitcoin3_ret_5d", "Retail_Sales_zscore_60d", "EWL_Switzerland_vol_20d", "PLD_Prologis_ret_5d", "CCI_CrownCastle_vol_20d"], "is_new": true}, {"model_id": "new_h3_NORMAL_XGBoost_N8_t5", "algo": "XGBoost", "regime": "NORMAL", "horizon": 3, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "IYM_BasicMaterials_ret_20d", "PAYX_Paychex_zscore_60d", "US7Y_Rate_ret_20d", "Core_PCE_zscore_60d", "DE_Deere_ret_5d", "AORD_AUS_zscore_60d"], "is_new": true}, {"model_id": "new_h3_NORMAL_XGBoost_N8_t6", "algo": "XGBoost", "regime": "NORMAL", "horizon": 3, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "vix_mean_abs_ret_5d", "AMZN_ret_5d", "EMR_Emerson_ret_20d", "TM_Telephone_ret_1d", "XOM_ret_1d", "HD_ret_1d"], "is_new": true}, {"model_id": "new_h3_NORMAL_XGBoost_N8_t7", "algo": "XGBoost", "regime": "NORMAL", "horizon": 3, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "VOD_Vodafone_zscore_60d", "PAYX_Paychex_vol_20d", "Michigan_Sentiment_ret_20d", "US3Y_Rate_ret_5d", "HD_ret_1d", "Core_PCE_zscore_60d"], "is_new": true}, {"model_id": "new_h3_NORMAL_XGBoost_N10_t0", "algo": "XGBoost", "regime": "NORMAL", "horizon": 3, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "GE_ret_1d", "TXN_vol_20d", "T10Y2Y_Spread_ret_5d", "Nikkei_Japan_vol_20d", "SLB_Schlumberger_ret_1d", "XLB_Materials_zscore_60d", "US7Y_Rate_ret_20d", "MS_MorganStanley_ret_5d"], "is_new": true}, {"model_id": "new_h3_NORMAL_XGBoost_N10_t1", "algo": "XGBoost", "regime": "NORMAL", "horizon": 3, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWM_Malaysia_zscore_60d", "US3M_Rate_vol_20d", "CCI_CrownCastle_vol_20d", "HangSeng_HK_ret_1d", "DAX_Germany_vol_20d", "TM_Telephone_vol_20d", "EWQ_France_zscore_60d", "ITT_ITTInc_ret_5d"], "is_new": true}, {"model_id": "new_h3_NORMAL_XGBoost_N10_t2", "algo": "XGBoost", "regime": "NORMAL", "horizon": 3, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "HD_ret_20d", "HD_zscore_60d", "EXC_Exelon_zscore_60d", "PLD_Prologis_ret_5d", "HD_ret_1d", "ES_Evergy_ret_1d", "PFE_ret_1d", "BA_ret_1d"], "is_new": true}, {"model_id": "new_h3_NORMAL_XGBoost_N10_t3", "algo": "XGBoost", "regime": "NORMAL", "horizon": 3, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "GE_ret_1d", "JNJ_ret_1d", "DOW_Price_zscore_60d", "Retail_Sales_zscore_60d", "US30Y_Rate_ret_20d", "NFCI_ret_5d", "BTI_BritishAmerican_ret_20d", "BLK_BlackRock_zscore_60d"], "is_new": true}, {"model_id": "new_h3_NORMAL_XGBoost_N10_t4", "algo": "XGBoost", "regime": "NORMAL", "horizon": 3, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "hmm_p_stress", "EMR_Emerson_ret_20d", "T_ret_1d", "QQQ_vol_20d", "GILD_Gilead_ret_20d", "BDX_Becton_Dickinson_ret_20d", "PCAR_PaccarInc_ret_5d", "INTC_ret_5d"], "is_new": true}, {"model_id": "new_h3_NORMAL_XGBoost_N10_t5", "algo": "XGBoost", "regime": "NORMAL", "horizon": 3, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "MSTR_Bitcoin3_ret_20d", "3M_vol_20d", "EWQ_France_zscore_60d", "PAYX_Paychex_vol_20d", "TXN_vol_20d", "EOG_EOGResources_ret_5d", "US30Y_Rate_ret_20d", "AORD_AUS_zscore_60d"], "is_new": true}, {"model_id": "new_h3_NORMAL_XGBoost_N10_t6", "algo": "XGBoost", "regime": "NORMAL", "horizon": 3, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "AMGN_Amgen_ret_1d", "US30Y_Rate_ret_20d", "hmm_p_stress", "heston_var_ev_h7", "MRK_Merck_zscore_60d", "TGT_Target_zscore_60d", "SCHW_Schwab_ret_5d", "Retail_Sales_zscore_60d"], "is_new": true}, {"model_id": "new_h3_NORMAL_XGBoost_N10_t7", "algo": "XGBoost", "regime": "NORMAL", "horizon": 3, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "AORD_AUS_zscore_60d", "CPB_CampbellSoup_zscore_60d", "EWA_Australia_ret_1d", "US5Y_Rate_ret_5d", "spx_abs_ret_max_5d", "XOM_ret_1d", "EWL_Switzerland_vol_20d", "WTI_Oil_FRED_zscore_60d"], "is_new": true}, {"model_id": "new_h3_NORMAL_XGBoost_N12_t0", "algo": "XGBoost", "regime": "NORMAL", "horizon": 3, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "PAYX_Paychex_zscore_60d", "XLK_Tech_zscore_60d", "Retail_Sales_zscore_60d", "HangSeng_HK_ret_5d", "MS_MorganStanley_ret_5d", "EWS_Singapore_ret_5d", "CMCSA_ret_1d", "SBUX_ret_5d", "BTI_BritishAmerican_ret_5d", "SPY_zscore_60d"], "is_new": true}, {"model_id": "new_h3_NORMAL_XGBoost_N12_t1", "algo": "XGBoost", "regime": "NORMAL", "horizon": 3, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "ORCL_zscore_60d", "TED_Spread_zscore_60d", "NFCI_ret_5d", "BTI_BritishAmerican_ret_5d", "Michigan_Sentiment_ret_20d", "EWM_Malaysia_zscore_60d", "EWM_Malaysia_ret_1d", "US30Y_Rate_ret_20d", "MS_MorganStanley_ret_5d", "EWH_HongKong_ret_5d"], "is_new": true}, {"model_id": "new_h3_NORMAL_XGBoost_N12_t2", "algo": "XGBoost", "regime": "NORMAL", "horizon": 3, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWC_Canada_zscore_60d", "DHR_vol_20d", "NEE_NextEra_ret_20d", "DE_Deere_ret_5d", "HangSeng_HK_ret_5d", "XLV_Health_zscore_60d", "SPY_zscore_60d", "NVDA_vol_20d", "AMZN_ret_5d", "XLK_Tech_zscore_60d"], "is_new": true}, {"model_id": "new_h3_NORMAL_XGBoost_N12_t3", "algo": "XGBoost", "regime": "NORMAL", "horizon": 3, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "GILD_Gilead_ret_20d", "US3M_Rate_zscore_60d", "SBUX_vol_20d", "PAYX_Paychex_zscore_60d", "AORD_AUS_zscore_60d", "INTC_ret_1d", "EWM_Malaysia_ret_1d", "MS_MorganStanley_ret_5d", "HangSeng_HK_ret_1d", "GD_GeneralDynamics_zscore_60d"], "is_new": true}, {"model_id": "new_h3_NORMAL_XGBoost_N12_t4", "algo": "XGBoost", "regime": "NORMAL", "horizon": 3, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "CPB_CampbellSoup_vol_20d", "DE_Deere_ret_5d", "CPB_CampbellSoup_zscore_60d", "spx_abs_ret_max_5d", "PLD_Prologis_ret_5d", "MSTR_Bitcoin3_ret_20d", "EOG_EOGResources_ret_5d", "AXP_Amex_vol_20d", "TM_Telephone_vol_20d", "hmm_p_stress"], "is_new": true}, {"model_id": "new_h3_NORMAL_XGBoost_N12_t5", "algo": "XGBoost", "regime": "NORMAL", "horizon": 3, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EXC_Exelon_ret_1d", "SLB_Schlumberger_ret_1d", "EMR_Emerson_ret_20d", "TGT_Target_zscore_60d", "AMD_ret_1d", "AMZN_ret_5d", "VOD_Vodafone_zscore_60d", "AMGN_Amgen_ret_1d", "LMT_LockheedMartin_vol_20d", "EWY_Korea_ret_20d"], "is_new": true}, {"model_id": "new_h3_NORMAL_XGBoost_N12_t6", "algo": "XGBoost", "regime": "NORMAL", "horizon": 3, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "heston_ev_h3", "LMT_LockheedMartin_vol_20d", "M_Macys_vol_20d", "TED_Spread_vol_20d", "Nikkei_Japan_vol_20d", "XLY_Disc_vol_20d", "NVDA_vol_20d", "WTI_Oil_FRED_zscore_60d", "AVB_AvalonBay_zscore_60d", "ASX_Australia_vol_20d"], "is_new": true}, {"model_id": "new_h3_NORMAL_XGBoost_N12_t7", "algo": "XGBoost", "regime": "NORMAL", "horizon": 3, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "CLX_Clorox_vol_20d", "INTC_ret_1d", "DE_Deere_vol_20d", "HangSeng_HK_ret_1d", "AMD_ret_5d", "GE_ret_1d", "DAX_Germany_vol_20d", "heston_var_ev_h3", "PCAR_PaccarInc_ret_5d", "IWM_SmallCap_vol_20d"], "is_new": true}, {"model_id": "new_h3_NORMAL_XGBoost_N15_t0", "algo": "XGBoost", "regime": "NORMAL", "horizon": 3, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "NWL_Newell_ret_20d", "SPY_zscore_60d", "NOC_Northrop_ret_20d", "CMCSA_ret_1d", "hmm_p_stress", "EOG_EOGResources_vol_20d", "Retail_Sales_zscore_60d", "PAYX_Paychex_vol_20d", "US3Y_Rate_ret_5d", "EWC_Canada_zscore_60d", "EWQ_France_zscore_60d", "EWM_Malaysia_ret_1d", "IYM_BasicMaterials_ret_20d"], "is_new": true}, {"model_id": "new_h3_NORMAL_XGBoost_N15_t1", "algo": "XGBoost", "regime": "NORMAL", "horizon": 3, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "CMCSA_ret_1d", "US1Y_Rate_ret_20d", "GD_GeneralDynamics_zscore_60d", "EXC_Exelon_ret_1d", "ITT_ITTInc_ret_5d", "PAYX_Paychex_vol_20d", "MS_MorganStanley_ret_5d", "MSTR_Bitcoin3_ret_1d", "CI_Cigna_vol_20d", "NOC_Northrop_ret_20d", "vix_mean_abs_ret_5d", "EOG_EOGResources_vol_20d", "US5Y_Rate_ret_5d"], "is_new": true}, {"model_id": "new_h3_NORMAL_XGBoost_N15_t2", "algo": "XGBoost", "regime": "NORMAL", "horizon": 3, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "HD_ret_20d", "BTI_BritishAmerican_ret_20d", "AMD_ret_5d", "EWJ_Japan_vol_20d", "Brent_Oil_FRED_ret_5d", "TGT_Target_zscore_60d", "HangSeng_HK_vol_20d", "heston_ev_h3", "hmm_p_stress", "SBUX_vol_20d", "EWS_Singapore_ret_5d", "vix_mean_abs_ret_5d", "EQR_Equity_ret_1d"], "is_new": true}, {"model_id": "new_h3_NORMAL_XGBoost_N15_t3", "algo": "XGBoost", "regime": "NORMAL", "horizon": 3, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWJ_Japan_vol_20d", "EXC_Exelon_ret_1d", "heston_ev_h3", "Nikkei_Japan_vol_20d", "MO_AltriaMG_ret_1d", "LOW_Lowes_ret_5d", "HD_ret_20d", "BTI_BritishAmerican_ret_20d", "DAX_Germany_zscore_60d", "spx_momentum_3d", "BLK_BlackRock_zscore_60d", "EWL_Switzerland_zscore_60d", "TM_Telephone_vol_20d"], "is_new": true}, {"model_id": "new_h3_NORMAL_XGBoost_N15_t4", "algo": "XGBoost", "regime": "NORMAL", "horizon": 3, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "ORCL_vol_20d", "EXC_Exelon_zscore_60d", "HangSeng_HK_ret_5d", "BA_ret_1d", "Brent_Oil_FRED_ret_5d", "XLB_Materials_zscore_60d", "SO_SouthernCo_ret_5d", "XLY_Disc_vol_20d", "EOG_EOGResources_vol_20d", "SLB_Schlumberger_ret_5d", "Core_CPI_zscore_60d", "HangSeng_HK_ret_1d", "EWH_HongKong_ret_5d"], "is_new": true}, {"model_id": "new_h3_NORMAL_XGBoost_N15_t5", "algo": "XGBoost", "regime": "NORMAL", "horizon": 3, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EQR_Equity_ret_1d", "JNJ_ret_1d", "gjr_condvar_h1", "Nikkei_Japan_zscore_60d", "DAX_Germany_zscore_60d", "TGT_Target_zscore_60d", "vix_mean_abs_ret_5d", "NWL_Newell_ret_20d", "heston_var_ev_h7", "3M_ret_5d", "SJM_JM_Smucker_ret_1d", "SLB_Schlumberger_ret_5d", "CI_Cigna_vol_20d"], "is_new": true}, {"model_id": "new_h3_NORMAL_XGBoost_N15_t6", "algo": "XGBoost", "regime": "NORMAL", "horizon": 3, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "Brent_Oil_FRED_ret_20d", "EWS_Singapore_ret_5d", "EXC_Exelon_ret_1d", "AXP_Amex_ret_20d", "PG_ret_20d", "GE_ret_1d", "XLY_Disc_vol_20d", "CLX_Clorox_vol_20d", "SJM_JM_Smucker_ret_5d", "INTC_ret_5d", "AMGN_Amgen_ret_1d", "MRK_Merck_zscore_60d", "TM_Telephone_ret_1d"], "is_new": true}, {"model_id": "new_h3_NORMAL_XGBoost_N15_t7", "algo": "XGBoost", "regime": "NORMAL", "horizon": 3, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "vix_mean_abs_ret_5d", "INTC_ret_1d", "GD_GeneralDynamics_zscore_60d", "NOC_Northrop_ret_20d", "LMT_LockheedMartin_ret_1d", "Brent_Oil_FRED_ret_5d", "DE_Deere_ret_5d", "BLK_BlackRock_zscore_60d", "M_Macys_vol_20d", "XLV_Health_zscore_60d", "AMT_AmericanTower_ret_1d", "spx_vol_5d", "XOM_ret_20d"], "is_new": true}, {"model_id": "new_h3_NORMAL_XGBoost_N20_t0", "algo": "XGBoost", "regime": "NORMAL", "horizon": 3, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "XOM_ret_1d", "LMT_LockheedMartin_ret_1d", "VRP_ma5", "PCAR_PaccarInc_ret_5d", "EWL_Switzerland_vol_20d", "heston_var_ev_h3", "ORCL_zscore_60d", "EFFR_ret_1d", "IYM_BasicMaterials_ret_20d", "EWQ_France_zscore_60d", "3M_ret_5d", "EWM_Malaysia_zscore_60d", "Core_PCE_zscore_60d", "INTC_ret_1d", "TGT_Target_zscore_60d", "GE_ret_1d", "MSTR_Bitcoin3_ret_1d", "IBEX_Spain_ret_20d"], "is_new": true}, {"model_id": "new_h3_NORMAL_XGBoost_N20_t1", "algo": "XGBoost", "regime": "NORMAL", "horizon": 3, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "T_ret_1d", "AMGN_Amgen_ret_1d", "ITT_ITTInc_ret_5d", "EXC_Exelon_ret_1d", "BLK_BlackRock_zscore_60d", "US7Y_Rate_ret_20d", "ENB_EnbridgeInc_ret_1d", "EWH_HongKong_ret_5d", "CPB_CampbellSoup_zscore_60d", "US5Y_Rate_ret_5d", "LMT_LockheedMartin_vol_20d", "AXP_Amex_vol_20d", "PPL_PPL_ret_1d", "US30Y_Rate_ret_20d", "SJM_JM_Smucker_ret_1d", "EWS_Singapore_ret_5d", "AMD_ret_5d", "JNJ_ret_1d"], "is_new": true}, {"model_id": "new_h3_NORMAL_XGBoost_N20_t2", "algo": "XGBoost", "regime": "NORMAL", "horizon": 3, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWQ_France_zscore_60d", "GE_ret_1d", "heston_ev_h3", "gjr_condvar_h1", "EWH_HongKong_ret_5d", "ES_Evergy_ret_1d", "XOM_ret_1d", "EFFR_vol_20d", "MS_MorganStanley_ret_1d", "DHR_vol_20d", "AMZN_ret_5d", "IYM_BasicMaterials_ret_20d", "LUV_SouthwestAir_ret_5d", "MO_AltriaMG_ret_1d", "LLY_zscore_60d", "MSTR_Bitcoin3_ret_20d", "DIS_vol_20d", "SLB_Schlumberger_ret_1d"], "is_new": true}, {"model_id": "new_h3_NORMAL_XGBoost_N20_t3", "algo": "XGBoost", "regime": "NORMAL", "horizon": 3, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "LLY_zscore_60d", "PAYX_Paychex_ret_20d", "TXN_vol_20d", "EWL_Switzerland_zscore_60d", "EWM_Malaysia_vol_20d", "VVIX_ret_20d", "EWS_Singapore_ret_5d", "XLB_Materials_zscore_60d", "US6M_Rate_ret_20d", "HD_zscore_60d", "EWH_HongKong_ret_5d", "Brent_Oil_FRED_ret_20d", "AORD_AUS_zscore_60d", "LMT_LockheedMartin_ret_1d", "CCI_CrownCastle_vol_20d", "TM_Telephone_vol_20d", "SLB_Schlumberger_ret_1d", "SLB_Schlumberger_ret_5d"], "is_new": true}, {"model_id": "new_h3_NORMAL_XGBoost_N20_t4", "algo": "XGBoost", "regime": "NORMAL", "horizon": 3, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "TED_Spread_zscore_60d", "AMD_ret_5d", "US3M_Rate_vol_20d", "JNJ_ret_1d", "SBUX_ret_5d", "EQR_Equity_ret_1d", "SLB_Schlumberger_ret_5d", "EWS_Singapore_ret_5d", "XLB_Materials_zscore_60d", "PAYX_Paychex_vol_20d", "SPY_zscore_60d", "EWC_Canada_zscore_60d", "Brent_Oil_FRED_ret_20d", "US30Y_Rate_ret_20d", "Nikkei_Japan_zscore_60d", "LOW_Lowes_ret_5d", "gjr_condvar_h1", "heston_var_ev_h5"], "is_new": true}, {"model_id": "new_h3_NORMAL_XGBoost_N20_t5", "algo": "XGBoost", "regime": "NORMAL", "horizon": 3, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "PAYX_Paychex_zscore_60d", "HD_zscore_60d", "JNJ_ret_1d", "NVDA_vol_20d", "EFFR_vol_20d", "vix_mean_abs_ret_5d", "INTC_ret_5d", "MSTR_Bitcoin3_ret_1d", "heston_var_ev_h3", "PLD_Prologis_ret_5d", "3M_ret_5d", "EWH_HongKong_ret_5d", "PAYX_Paychex_vol_20d", "CPB_CampbellSoup_ret_5d", "TGT_Target_zscore_60d", "SLB_Schlumberger_ret_1d", "LOW_Lowes_ret_20d", "T10Y2Y_Spread_ret_5d"], "is_new": true}, {"model_id": "new_h3_NORMAL_XGBoost_N20_t6", "algo": "XGBoost", "regime": "NORMAL", "horizon": 3, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "vix_acceleration_1d", "DE_Deere_ret_5d", "ASX_Australia_ret_5d", "SBUX_vol_20d", "JNJ_ret_1d", "spx_vol_5d", "DAX_Germany_zscore_60d", "AXP_Amex_ret_20d", "EWY_Korea_zscore_60d", "M_Macys_vol_20d", "US7Y_Rate_ret_20d", "AXP_Amex_vol_20d", "PG_ret_20d", "HD_ret_5d", "EWH_HongKong_ret_5d", "US6M_Rate_ret_20d", "AMT_AmericanTower_ret_1d", "gjr_condvar_h1"], "is_new": true}, {"model_id": "new_h3_NORMAL_XGBoost_N20_t7", "algo": "XGBoost", "regime": "NORMAL", "horizon": 3, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "SCHW_Schwab_ret_5d", "EWA_Australia_ret_1d", "XLV_Health_zscore_60d", "ENB_EnbridgeInc_ret_1d", "AVB_AvalonBay_zscore_60d", "DOW_Price_zscore_60d", "HangSeng_HK_ret_1d", "gjr_condvar_h1", "EXC_Exelon_ret_1d", "GD_GeneralDynamics_zscore_60d", "AMD_ret_5d", "Retail_Sales_zscore_60d", "EWQ_France_ret_20d", "heston_ev_h3", "CPB_CampbellSoup_ret_5d", "US3Y_Rate_ret_5d", "SBUX_ret_5d", "CCI_CrownCastle_vol_20d"], "is_new": true}, {"model_id": "new_h3_NORMAL_XGBoost_N25_t0", "algo": "XGBoost", "regime": "NORMAL", "horizon": 3, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EQR_Equity_ret_1d", "BDX_Becton_Dickinson_ret_20d", "EWC_Canada_zscore_60d", "MS_MorganStanley_zscore_60d", "SBUX_zscore_60d", "vix_mean_abs_ret_5d", "XLK_Tech_zscore_60d", "Brent_Oil_FRED_ret_20d", "spx_momentum_3d", "XLY_Disc_vol_20d", "gjr_condvar_h1", "EWY_Korea_zscore_60d", "DAX_Germany_zscore_60d", "Nikkei_Japan_zscore_60d", "CI_Cigna_vol_20d", "MS_MorganStanley_ret_5d", "NOC_Northrop_ret_20d", "NVDA_vol_20d", "GILD_Gilead_ret_20d", "SBUX_ret_5d", "BTI_BritishAmerican_ret_20d", "AORD_AUS_zscore_60d", "spx_abs_ret_max_5d"], "is_new": true}, {"model_id": "new_h3_NORMAL_XGBoost_N25_t1", "algo": "XGBoost", "regime": "NORMAL", "horizon": 3, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "PAYX_Paychex_ret_20d", "JNJ_ret_1d", "NVDA_vol_20d", "LUV_SouthwestAir_ret_5d", "DE_Deere_vol_20d", "AMZN_ret_5d", "TXN_vol_20d", "EWY_Korea_ret_20d", "CPB_CampbellSoup_zscore_60d", "EWA_Australia_zscore_60d", "AMD_ret_1d", "3M_vol_20d", "Brent_Oil_FRED_ret_5d", "MSTR_Bitcoin3_ret_5d", "CCI_CrownCastle_vol_20d", "TED_Spread_vol_20d", "EWL_Switzerland_zscore_60d", "EQIX_Equinix_ret_5d", "CMCSA_ret_1d", "IYM_BasicMaterials_ret_20d", "EFFR_vol_20d", "US3M_Rate_zscore_60d", "AXP_Amex_ret_20d"], "is_new": true}, {"model_id": "new_h3_NORMAL_XGBoost_N25_t2", "algo": "XGBoost", "regime": "NORMAL", "horizon": 3, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "MSTR_Bitcoin3_ret_5d", "MSTR_Bitcoin3_ret_1d", "GD_GeneralDynamics_zscore_60d", "EWH_HongKong_ret_5d", "ASX_Australia_vol_20d", "T_ret_1d", "vix_acceleration_1d", "AMZN_ret_5d", "HUM_Humana_ret_5d", "DIS_vol_20d", "US1Y_Rate_ret_5d", "EWQ_France_ret_20d", "CPB_CampbellSoup_ret_20d", "MS_MorganStanley_zscore_60d", "SBUX_zscore_60d", "DHR_ret_1d", "EWJ_Japan_vol_20d", "DE_Deere_ret_5d", "HangSeng_HK_ret_1d", "MSTR_Bitcoin3_ret_20d", "EWY_Korea_zscore_60d", "HD_ret_1d", "QQQ_vol_20d"], "is_new": true}, {"model_id": "new_h3_NORMAL_XGBoost_N25_t3", "algo": "XGBoost", "regime": "NORMAL", "horizon": 3, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "MS_MorganStanley_ret_5d", "BLK_BlackRock_zscore_60d", "Nikkei_Japan_zscore_60d", "PCAR_PaccarInc_ret_5d", "heston_var_ev_h5", "EWQ_France_ret_20d", "SBUX_zscore_60d", "spx_momentum_3d", "TED_Spread_zscore_60d", "TGT_Target_zscore_60d", "XLY_Disc_vol_20d", "ENB_EnbridgeInc_ret_1d", "Brent_Oil_FRED_ret_20d", "EXC_Exelon_ret_1d", "DAX_Germany_vol_20d", "TXN_vol_20d", "Core_PCE_zscore_60d", "AVB_AvalonBay_zscore_60d", "vix_acceleration_1d", "PFE_ret_1d", "EMR_Emerson_ret_20d", "AMZN_ret_5d", "SJM_JM_Smucker_ret_5d"], "is_new": true}, {"model_id": "new_h3_NORMAL_XGBoost_N25_t4", "algo": "XGBoost", "regime": "NORMAL", "horizon": 3, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "CPB_CampbellSoup_vol_20d", "spx_abs_ret_max_5d", "CPB_CampbellSoup_ret_20d", "PCAR_PaccarInc_ret_5d", "BLK_BlackRock_zscore_60d", "LLY_zscore_60d", "MS_MorganStanley_ret_5d", "GE_ret_1d", "PLD_Prologis_ret_5d", "Nikkei_Japan_vol_20d", "IYM_BasicMaterials_ret_20d", "TM_Telephone_vol_20d", "MO_AltriaMG_ret_1d", "spx_momentum_3d", "LUV_SouthwestAir_ret_5d", "PAYX_Paychex_zscore_60d", "VRP_ma5", "AMT_AmericanTower_ret_1d", "EWM_Malaysia_ret_1d", "WTI_Oil_FRED_zscore_60d", "M_Macys_vol_20d", "PG_ret_20d", "INTC_ret_5d"], "is_new": true}, {"model_id": "new_h3_NORMAL_XGBoost_N25_t5", "algo": "XGBoost", "regime": "NORMAL", "horizon": 3, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "AMT_AmericanTower_ret_1d", "US7Y_Rate_ret_20d", "XLY_Disc_vol_20d", "spx_vol_5d", "vix_acceleration_1d", "XOM_ret_20d", "EWL_Switzerland_vol_20d", "SBUX_ret_5d", "TED_Spread_zscore_60d", "HD_ret_20d", "TM_Telephone_vol_20d", "M_Macys_vol_20d", "INTC_ret_1d", "EMR_Emerson_ret_20d", "heston_var_ev_h7", "AXP_Amex_ret_20d", "TM_Telephone_ret_1d", "3M_vol_20d", "HD_ret_5d", "XOM_ret_1d", "EWJ_Japan_vol_20d", "EWA_Australia_zscore_60d", "FedFunds_zscore_60d"], "is_new": true}, {"model_id": "new_h3_NORMAL_XGBoost_N25_t6", "algo": "XGBoost", "regime": "NORMAL", "horizon": 3, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "AMZN_ret_5d", "PG_ret_20d", "ASX_Australia_ret_5d", "3M_ret_5d", "vix_mean_abs_ret_5d", "EFFR_ret_1d", "MS_MorganStanley_ret_5d", "PLD_Prologis_ret_5d", "LMT_LockheedMartin_ret_1d", "Industrial_Production_zscore_60d", "FedFunds_zscore_60d", "PAYX_Paychex_vol_20d", "LLY_zscore_60d", "EWG_Germany_ret_20d", "XLV_Health_zscore_60d", "ORCL_vol_20d", "HD_zscore_60d", "CLX_Clorox_vol_20d", "NWL_Newell_ret_20d", "EWH_HongKong_ret_5d", "TED_Spread_zscore_60d", "DHR_ret_1d", "JNJ_ret_1d"], "is_new": true}, {"model_id": "new_h3_NORMAL_XGBoost_N25_t7", "algo": "XGBoost", "regime": "NORMAL", "horizon": 3, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "heston_ev_h3", "EWG_Germany_ret_20d", "BA_ret_1d", "EWA_Australia_zscore_60d", "LMT_LockheedMartin_ret_1d", "SLB_Schlumberger_ret_1d", "EMR_Emerson_ret_20d", "CLX_Clorox_vol_20d", "QQQ_vol_20d", "EOG_EOGResources_vol_20d", "MS_MorganStanley_ret_1d", "EWM_Malaysia_ret_1d", "EWY_Korea_ret_20d", "AORD_AUS_zscore_60d", "TXN_vol_20d", "spx_vol_5d", "EXC_Exelon_zscore_60d", "US1Y_Rate_ret_20d", "AMD_ret_5d", "TM_Telephone_ret_1d", "SO_SouthernCo_ret_5d", "Nikkei_Japan_vol_20d", "CCI_CrownCastle_vol_20d"], "is_new": true}, {"model_id": "new_h3_NORMAL_XGBoost_N30_t0", "algo": "XGBoost", "regime": "NORMAL", "horizon": 3, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "HangSeng_HK_ret_5d", "vix_mean_abs_ret_5d", "EWM_Malaysia_zscore_60d", "BLK_BlackRock_zscore_60d", "MS_MorganStanley_zscore_60d", "INTC_ret_5d", "PG_ret_20d", "SBUX_ret_5d", "DAX_Germany_zscore_60d", "CMCSA_ret_1d", "CPB_CampbellSoup_ret_5d", "DE_Deere_vol_20d", "PAYX_Paychex_ret_20d", "CPB_CampbellSoup_zscore_60d", "Michigan_Sentiment_ret_20d", "SPY_zscore_60d", "XLY_Disc_vol_20d", "3M_ret_5d", "US3M_Rate_vol_20d", "TED_Spread_zscore_60d", "CPB_CampbellSoup_ret_20d", "PCAR_PaccarInc_ret_5d", "NOC_Northrop_ret_20d", "Nikkei_Japan_vol_20d", "spx_vol_5d", "MO_AltriaMG_ret_1d", "heston_var_ev_h3", "CI_Cigna_vol_20d"], "is_new": true}, {"model_id": "new_h3_NORMAL_XGBoost_N30_t1", "algo": "XGBoost", "regime": "NORMAL", "horizon": 3, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWG_Germany_ret_20d", "spx_abs_ret_max_5d", "PAYX_Paychex_zscore_60d", "Core_CPI_zscore_60d", "Brent_Oil_FRED_ret_20d", "AMD_ret_5d", "spx_vol_5d", "3M_vol_20d", "MS_MorganStanley_zscore_60d", "TGT_Target_zscore_60d", "NFCI_ret_5d", "TED_Spread_vol_20d", "Nikkei_Japan_zscore_60d", "GILD_Gilead_ret_20d", "NVDA_vol_20d", "HangSeng_HK_vol_20d", "CPB_CampbellSoup_ret_5d", "HD_zscore_60d", "QQQ_vol_20d", "EWQ_France_ret_20d", "LMT_LockheedMartin_ret_1d", "IBEX_Spain_ret_20d", "Nikkei_Japan_vol_20d", "T_ret_1d", "US3M_Rate_zscore_60d", "IYR_US_REIT2_zscore_60d", "EWA_Australia_zscore_60d", "PLD_Prologis_ret_5d"], "is_new": true}, {"model_id": "new_h3_NORMAL_XGBoost_N30_t2", "algo": "XGBoost", "regime": "NORMAL", "horizon": 3, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "INTC_ret_5d", "PFE_ret_1d", "PAYX_Paychex_ret_20d", "MSTR_Bitcoin3_ret_20d", "TM_Telephone_ret_1d", "US7Y_Rate_ret_20d", "US3M_Rate_zscore_60d", "heston_var_ev_h3", "SBUX_zscore_60d", "SBUX_ret_5d", "EWL_Switzerland_zscore_60d", "US3Y_Rate_ret_5d", "MRK_Merck_zscore_60d", "CLX_Clorox_vol_20d", "MS_MorganStanley_ret_1d", "MSTR_Bitcoin3_ret_5d", "ASX_Australia_ret_5d", "EWM_Malaysia_ret_1d", "EMR_Emerson_ret_20d", "EWY_Korea_ret_20d", "Retail_Sales_zscore_60d", "AORD_AUS_zscore_60d", "CPB_CampbellSoup_ret_5d", "FedFunds_zscore_60d", "AMGN_Amgen_ret_1d", "LMT_LockheedMartin_vol_20d", "BLK_BlackRock_zscore_60d", "ORCL_zscore_60d"], "is_new": true}, {"model_id": "new_h3_NORMAL_XGBoost_N30_t3", "algo": "XGBoost", "regime": "NORMAL", "horizon": 3, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "XLY_Disc_vol_20d", "XLK_Tech_zscore_60d", "MSTR_Bitcoin3_ret_20d", "gjr_condvar_h1", "EWL_Switzerland_zscore_60d", "DHR_vol_20d", "EWM_Malaysia_ret_1d", "BDX_Becton_Dickinson_ret_20d", "HangSeng_HK_vol_20d", "SBUX_vol_20d", "SBUX_zscore_60d", "DOW_Price_zscore_60d", "Industrial_Production_zscore_60d", "VRP_ma5", "NVDA_vol_20d", "Brent_Oil_FRED_ret_5d", "CMCSA_ret_1d", "LOW_Lowes_ret_5d", "LLY_zscore_60d", "EWM_Malaysia_vol_20d", "VVIX_ret_20d", "SLB_Schlumberger_ret_1d", "HD_zscore_60d", "XLF_Fin_vol_20d", "MS_MorganStanley_ret_5d", "DAX_Germany_zscore_60d", "AMGN_Amgen_ret_1d", "ORCL_vol_20d"], "is_new": true}, {"model_id": "new_h3_NORMAL_XGBoost_N30_t4", "algo": "XGBoost", "regime": "NORMAL", "horizon": 3, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "MS_MorganStanley_zscore_60d", "BA_ret_1d", "spx_vol_5d", "EWA_Australia_zscore_60d", "AMZN_ret_5d", "US5Y_Rate_ret_5d", "TGT_Target_zscore_60d", "BDX_Becton_Dickinson_ret_20d", "LMT_LockheedMartin_ret_1d", "CI_Cigna_vol_20d", "T10Y2Y_Spread_ret_5d", "EWL_Switzerland_vol_20d", "FedFunds_zscore_60d", "EWG_Germany_ret_20d", "GILD_Gilead_ret_20d", "US3M_Rate_zscore_60d", "EWG_Germany_vol_20d", "NEE_NextEra_ret_20d", "EWY_Korea_ret_20d", "ITT_ITTInc_ret_5d", "ORCL_vol_20d", "TED_Spread_zscore_60d", "BLK_BlackRock_zscore_60d", "gjr_condvar_h1", "SJM_JM_Smucker_ret_1d", "PAYX_Paychex_ret_20d", "WTI_Oil_FRED_zscore_60d", "HD_ret_20d"], "is_new": true}, {"model_id": "new_h3_NORMAL_XGBoost_N30_t5", "algo": "XGBoost", "regime": "NORMAL", "horizon": 3, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "CPB_CampbellSoup_ret_20d", "NWL_Newell_ret_20d", "PG_ret_20d", "MS_MorganStanley_ret_1d", "DHR_vol_20d", "vix_mean_abs_ret_5d", "TM_Telephone_ret_1d", "SBUX_zscore_60d", "HD_zscore_60d", "EWQ_France_zscore_60d", "AMT_AmericanTower_ret_1d", "XLY_Disc_vol_20d", "NEE_NextEra_ret_20d", "HD_ret_5d", "SJM_JM_Smucker_ret_1d", "HangSeng_HK_ret_5d", "DE_Deere_ret_5d", "XLB_Materials_zscore_60d", "EWG_Germany_ret_20d", "DHR_ret_1d", "MS_MorganStanley_ret_5d", "TED_Spread_zscore_60d", "EWQ_France_ret_20d", "XLK_Tech_zscore_60d", "T10Y2Y_Spread_ret_5d", "SBUX_ret_5d", "AMZN_ret_5d", "EOG_EOGResources_vol_20d"], "is_new": true}, {"model_id": "new_h3_NORMAL_XGBoost_N30_t6", "algo": "XGBoost", "regime": "NORMAL", "horizon": 3, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EOG_EOGResources_ret_5d", "US30Y_Rate_ret_20d", "VOD_Vodafone_zscore_60d", "AVB_AvalonBay_zscore_60d", "US1Y_Rate_ret_5d", "IBEX_Spain_ret_20d", "LMT_LockheedMartin_ret_1d", "FedFunds_zscore_60d", "TED_Spread_vol_20d", "EFFR_ret_1d", "MSTR_Bitcoin3_ret_1d", "DHR_ret_1d", "ORCL_zscore_60d", "Retail_Sales_zscore_60d", "EWG_Germany_vol_20d", "EWM_Malaysia_ret_1d", "CPB_CampbellSoup_vol_20d", "ES_Evergy_ret_1d", "Industrial_Production_zscore_60d", "T10Y2Y_Spread_ret_5d", "CPB_CampbellSoup_ret_20d", "DE_Deere_ret_5d", "vix_acceleration_1d", "PLD_Prologis_ret_5d", "MS_MorganStanley_ret_5d", "Michigan_Sentiment_ret_20d", "XLK_Tech_zscore_60d", "GD_GeneralDynamics_zscore_60d"], "is_new": true}, {"model_id": "new_h3_NORMAL_XGBoost_N30_t7", "algo": "XGBoost", "regime": "NORMAL", "horizon": 3, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWY_Korea_zscore_60d", "EXC_Exelon_zscore_60d", "PLD_Prologis_ret_5d", "NWL_Newell_ret_20d", "JNJ_ret_1d", "EQR_Equity_ret_1d", "DE_Deere_ret_5d", "HangSeng_HK_ret_5d", "M_Macys_vol_20d", "XLK_Tech_zscore_60d", "heston_var_ev_h3", "SPY_zscore_60d", "IYM_BasicMaterials_ret_20d", "Core_CPI_zscore_60d", "NFCI_ret_5d", "VRP_ma5", "EQIX_Equinix_ret_5d", "BTI_BritishAmerican_ret_5d", "Retail_Sales_zscore_60d", "MO_AltriaMG_ret_1d", "Brent_Oil_FRED_ret_20d", "PG_ret_20d", "CPB_CampbellSoup_vol_20d", "EWY_Korea_ret_20d", "MRK_Merck_zscore_60d", "DHR_vol_20d", "US1Y_Rate_ret_20d", "HangSeng_HK_ret_1d"], "is_new": true}, {"model_id": "new_h3_NORMAL_LightGBM_N5_t0", "algo": "LightGBM", "regime": "NORMAL", "horizon": 3, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "XOM_ret_20d", "IWM_SmallCap_vol_20d", "3M_vol_20d"], "is_new": true}, {"model_id": "new_h3_NORMAL_LightGBM_N5_t1", "algo": "LightGBM", "regime": "NORMAL", "horizon": 3, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EXC_Exelon_zscore_60d", "IYR_US_REIT2_zscore_60d", "TED_Spread_vol_20d"], "is_new": true}, {"model_id": "new_h3_NORMAL_LightGBM_N5_t2", "algo": "LightGBM", "regime": "NORMAL", "horizon": 3, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "IBEX_Spain_ret_20d", "TGT_Target_zscore_60d", "PFE_ret_1d"], "is_new": true}, {"model_id": "new_h3_NORMAL_LightGBM_N5_t3", "algo": "LightGBM", "regime": "NORMAL", "horizon": 3, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "TXN_vol_20d", "vix_acceleration_1d", "EWY_Korea_ret_20d"], "is_new": true}, {"model_id": "new_h3_NORMAL_LightGBM_N5_t4", "algo": "LightGBM", "regime": "NORMAL", "horizon": 3, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "AXP_Amex_ret_20d", "SCHW_Schwab_ret_5d", "TM_Telephone_ret_1d"], "is_new": true}, {"model_id": "new_h3_NORMAL_LightGBM_N5_t5", "algo": "LightGBM", "regime": "NORMAL", "horizon": 3, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "TM_Telephone_vol_20d", "EQIX_Equinix_ret_5d", "heston_var_ev_h3"], "is_new": true}, {"model_id": "new_h3_NORMAL_LightGBM_N5_t6", "algo": "LightGBM", "regime": "NORMAL", "horizon": 3, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "HD_ret_5d", "TM_Telephone_vol_20d", "TM_Telephone_ret_1d"], "is_new": true}, {"model_id": "new_h3_NORMAL_LightGBM_N5_t7", "algo": "LightGBM", "regime": "NORMAL", "horizon": 3, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "LUV_SouthwestAir_ret_5d", "EWY_Korea_ret_20d", "IBEX_Spain_ret_20d"], "is_new": true}, {"model_id": "new_h3_NORMAL_LightGBM_N8_t0", "algo": "LightGBM", "regime": "NORMAL", "horizon": 3, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "IBEX_Spain_ret_20d", "NEE_NextEra_ret_20d", "CPB_CampbellSoup_zscore_60d", "FedFunds_zscore_60d", "CPB_CampbellSoup_ret_5d", "CPB_CampbellSoup_vol_20d"], "is_new": true}, {"model_id": "new_h3_NORMAL_LightGBM_N8_t1", "algo": "LightGBM", "regime": "NORMAL", "horizon": 3, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "HD_zscore_60d", "PG_ret_20d", "EWL_Switzerland_zscore_60d", "US3M_Rate_zscore_60d", "heston_var_ev_h5", "spx_vol_5d"], "is_new": true}, {"model_id": "new_h3_NORMAL_LightGBM_N8_t2", "algo": "LightGBM", "regime": "NORMAL", "horizon": 3, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "INTC_ret_1d", "T10Y2Y_Spread_ret_5d", "EWY_Korea_zscore_60d", "HD_ret_20d", "US30Y_Rate_ret_20d", "AMD_ret_1d"], "is_new": true}, {"model_id": "new_h3_NORMAL_LightGBM_N8_t3", "algo": "LightGBM", "regime": "NORMAL", "horizon": 3, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "QQQ_vol_20d", "Brent_Oil_FRED_ret_20d", "DE_Deere_vol_20d", "PAYX_Paychex_ret_20d", "SBUX_zscore_60d", "CPB_CampbellSoup_vol_20d"], "is_new": true}, {"model_id": "new_h3_NORMAL_LightGBM_N8_t4", "algo": "LightGBM", "regime": "NORMAL", "horizon": 3, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "XLB_Materials_zscore_60d", "3M_vol_20d", "VRP_ma5", "CPB_CampbellSoup_ret_20d", "PAYX_Paychex_zscore_60d", "HD_ret_20d"], "is_new": true}, {"model_id": "new_h3_NORMAL_LightGBM_N8_t5", "algo": "LightGBM", "regime": "NORMAL", "horizon": 3, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "TED_Spread_vol_20d", "3M_ret_5d", "spx_momentum_3d", "EWA_Australia_zscore_60d", "AORD_AUS_zscore_60d", "TM_Telephone_ret_1d"], "is_new": true}, {"model_id": "new_h3_NORMAL_LightGBM_N8_t6", "algo": "LightGBM", "regime": "NORMAL", "horizon": 3, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "Michigan_Sentiment_ret_20d", "EXC_Exelon_zscore_60d", "CPB_CampbellSoup_ret_20d", "XLB_Materials_zscore_60d", "XLY_Disc_vol_20d", "MS_MorganStanley_ret_1d"], "is_new": true}, {"model_id": "new_h3_NORMAL_LightGBM_N8_t7", "algo": "LightGBM", "regime": "NORMAL", "horizon": 3, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "US3M_Rate_vol_20d", "EWS_Singapore_ret_5d", "PG_ret_20d", "NVDA_vol_20d", "EWY_Korea_zscore_60d", "ITT_ITTInc_ret_5d"], "is_new": true}, {"model_id": "new_h3_NORMAL_LightGBM_N10_t0", "algo": "LightGBM", "regime": "NORMAL", "horizon": 3, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "DE_Deere_ret_5d", "TED_Spread_vol_20d", "AMD_ret_5d", "EOG_EOGResources_ret_5d", "EWQ_France_ret_20d", "T_ret_1d", "MSTR_Bitcoin3_ret_5d", "INTC_ret_5d"], "is_new": true}, {"model_id": "new_h3_NORMAL_LightGBM_N10_t1", "algo": "LightGBM", "regime": "NORMAL", "horizon": 3, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "AMD_ret_5d", "US1Y_Rate_ret_20d", "AMD_ret_1d", "LUV_SouthwestAir_ret_5d", "US6M_Rate_ret_20d", "TXN_vol_20d", "AMZN_ret_5d", "vix_mean_abs_ret_5d"], "is_new": true}, {"model_id": "new_h3_NORMAL_LightGBM_N10_t2", "algo": "LightGBM", "regime": "NORMAL", "horizon": 3, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "3M_ret_5d", "IYR_US_REIT2_zscore_60d", "EWA_Australia_zscore_60d", "PCAR_PaccarInc_ret_5d", "TM_Telephone_ret_1d", "EFFR_ret_1d", "TXN_vol_20d", "MS_MorganStanley_zscore_60d"], "is_new": true}, {"model_id": "new_h3_NORMAL_LightGBM_N10_t3", "algo": "LightGBM", "regime": "NORMAL", "horizon": 3, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "BA_ret_1d", "EWA_Australia_ret_1d", "AMD_ret_1d", "PCAR_PaccarInc_ret_5d", "Brent_Oil_FRED_ret_20d", "BLK_BlackRock_zscore_60d", "TM_Telephone_vol_20d", "BTI_BritishAmerican_ret_20d"], "is_new": true}, {"model_id": "new_h3_NORMAL_LightGBM_N10_t4", "algo": "LightGBM", "regime": "NORMAL", "horizon": 3, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "VRP_ma5", "PLD_Prologis_ret_5d", "MSTR_Bitcoin3_ret_1d", "heston_var_ev_h7", "MS_MorganStanley_ret_5d", "ENB_EnbridgeInc_ret_1d", "CTAS_Cintas_vol_20d", "LUV_SouthwestAir_ret_5d"], "is_new": true}, {"model_id": "new_h3_NORMAL_LightGBM_N10_t5", "algo": "LightGBM", "regime": "NORMAL", "horizon": 3, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "US6M_Rate_ret_20d", "US30Y_Rate_ret_20d", "BLK_BlackRock_zscore_60d", "PAYX_Paychex_vol_20d", "3M_ret_5d", "DAX_Germany_zscore_60d", "PCAR_PaccarInc_ret_5d", "DE_Deere_vol_20d"], "is_new": true}, {"model_id": "new_h3_NORMAL_LightGBM_N10_t6", "algo": "LightGBM", "regime": "NORMAL", "horizon": 3, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "spx_vol_5d", "US30Y_Rate_ret_20d", "NEE_NextEra_ret_20d", "EWL_Switzerland_vol_20d", "IYR_US_REIT2_zscore_60d", "TM_Telephone_ret_1d", "EWQ_France_ret_20d", "HangSeng_HK_vol_20d"], "is_new": true}, {"model_id": "new_h3_NORMAL_LightGBM_N10_t7", "algo": "LightGBM", "regime": "NORMAL", "horizon": 3, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "US30Y_Rate_ret_20d", "US7Y_Rate_ret_20d", "heston_var_ev_h3", "TM_Telephone_vol_20d", "SBUX_vol_20d", "heston_var_ev_h7", "spx_abs_ret_max_5d", "GD_GeneralDynamics_zscore_60d"], "is_new": true}, {"model_id": "new_h3_NORMAL_LightGBM_N12_t0", "algo": "LightGBM", "regime": "NORMAL", "horizon": 3, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "BA_ret_1d", "XLK_Tech_zscore_60d", "DHR_vol_20d", "EWM_Malaysia_vol_20d", "EWQ_France_zscore_60d", "LOW_Lowes_ret_5d", "TED_Spread_vol_20d", "SLB_Schlumberger_ret_1d", "LMT_LockheedMartin_ret_1d", "PCAR_PaccarInc_ret_5d"], "is_new": true}, {"model_id": "new_h3_NORMAL_LightGBM_N12_t1", "algo": "LightGBM", "regime": "NORMAL", "horizon": 3, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "spx_abs_ret_max_5d", "EWL_Switzerland_vol_20d", "heston_var_ev_h7", "SPY_zscore_60d", "XLF_Fin_vol_20d", "CPB_CampbellSoup_ret_20d", "ORCL_vol_20d", "CCI_CrownCastle_vol_20d", "spx_vol_5d", "WTI_Oil_FRED_zscore_60d"], "is_new": true}, {"model_id": "new_h3_NORMAL_LightGBM_N12_t2", "algo": "LightGBM", "regime": "NORMAL", "horizon": 3, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "LUV_SouthwestAir_ret_5d", "EWH_HongKong_ret_5d", "AMGN_Amgen_ret_1d", "heston_ev_h3", "DOW_Price_zscore_60d", "VOD_Vodafone_zscore_60d", "NEE_NextEra_ret_20d", "CLX_Clorox_vol_20d", "GE_ret_1d", "US7Y_Rate_ret_20d"], "is_new": true}, {"model_id": "new_h3_NORMAL_LightGBM_N12_t3", "algo": "LightGBM", "regime": "NORMAL", "horizon": 3, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "Nikkei_Japan_vol_20d", "PLD_Prologis_ret_5d", "US3M_Rate_vol_20d", "IWM_SmallCap_vol_20d", "XOM_ret_20d", "spx_momentum_3d", "EFFR_vol_20d", "hmm_p_stress", "AVB_AvalonBay_zscore_60d", "ES_Evergy_ret_1d"], "is_new": true}, {"model_id": "new_h3_NORMAL_LightGBM_N12_t4", "algo": "LightGBM", "regime": "NORMAL", "horizon": 3, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "FedFunds_zscore_60d", "TM_Telephone_vol_20d", "CCI_CrownCastle_vol_20d", "Brent_Oil_FRED_ret_20d", "US30Y_Rate_ret_20d", "XLV_Health_zscore_60d", "Core_PCE_zscore_60d", "QQQ_vol_20d", "XOM_ret_20d", "EMR_Emerson_ret_20d"], "is_new": true}, {"model_id": "new_h3_NORMAL_LightGBM_N12_t5", "algo": "LightGBM", "regime": "NORMAL", "horizon": 3, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "TM_Telephone_ret_1d", "SBUX_ret_5d", "GE_ret_1d", "VOD_Vodafone_zscore_60d", "EXC_Exelon_ret_1d", "AVB_AvalonBay_zscore_60d", "AXP_Amex_ret_20d", "US5Y_Rate_ret_5d", "EOG_EOGResources_vol_20d", "NVDA_vol_20d"], "is_new": true}, {"model_id": "new_h3_NORMAL_LightGBM_N12_t6", "algo": "LightGBM", "regime": "NORMAL", "horizon": 3, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "SCHW_Schwab_ret_5d", "spx_momentum_3d", "ES_Evergy_ret_1d", "AMT_AmericanTower_ret_1d", "BDX_Becton_Dickinson_ret_20d", "US1Y_Rate_ret_5d", "heston_var_ev_h3", "spx_vol_5d", "LOW_Lowes_ret_20d", "PCAR_PaccarInc_ret_5d"], "is_new": true}, {"model_id": "new_h3_NORMAL_LightGBM_N12_t7", "algo": "LightGBM", "regime": "NORMAL", "horizon": 3, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWH_HongKong_ret_5d", "spx_momentum_3d", "vix_acceleration_1d", "ITT_ITTInc_ret_5d", "PAYX_Paychex_vol_20d", "DIS_vol_20d", "XLV_Health_zscore_60d", "DAX_Germany_zscore_60d", "IYM_BasicMaterials_ret_20d", "LLY_zscore_60d"], "is_new": true}, {"model_id": "new_h3_NORMAL_LightGBM_N15_t0", "algo": "LightGBM", "regime": "NORMAL", "horizon": 3, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "T_ret_1d", "GD_GeneralDynamics_zscore_60d", "NWL_Newell_ret_20d", "WTI_Oil_FRED_zscore_60d", "AMD_ret_5d", "3M_ret_5d", "CPB_CampbellSoup_vol_20d", "Industrial_Production_zscore_60d", "spx_abs_ret_max_5d", "US30Y_Rate_ret_20d", "CTAS_Cintas_vol_20d", "DAX_Germany_zscore_60d", "EWJ_Japan_vol_20d"], "is_new": true}, {"model_id": "new_h3_NORMAL_LightGBM_N15_t1", "algo": "LightGBM", "regime": "NORMAL", "horizon": 3, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "HUM_Humana_ret_5d", "XLY_Disc_vol_20d", "heston_var_ev_h7", "HD_ret_5d", "SBUX_ret_5d", "ASX_Australia_vol_20d", "3M_vol_20d", "DE_Deere_vol_20d", "LLY_zscore_60d", "Core_CPI_zscore_60d", "US3Y_Rate_ret_5d", "AVB_AvalonBay_zscore_60d", "MS_MorganStanley_ret_5d"], "is_new": true}, {"model_id": "new_h3_NORMAL_LightGBM_N15_t2", "algo": "LightGBM", "regime": "NORMAL", "horizon": 3, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "GD_GeneralDynamics_zscore_60d", "NWL_Newell_ret_20d", "AXP_Amex_vol_20d", "XOM_ret_1d", "LOW_Lowes_ret_5d", "QQQ_vol_20d", "SBUX_vol_20d", "EFFR_vol_20d", "HD_ret_5d", "TGT_Target_zscore_60d", "Retail_Sales_zscore_60d", "PFE_ret_1d", "MSTR_Bitcoin3_ret_5d"], "is_new": true}, {"model_id": "new_h3_NORMAL_LightGBM_N15_t3", "algo": "LightGBM", "regime": "NORMAL", "horizon": 3, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "ORCL_zscore_60d", "DE_Deere_ret_5d", "vix_mean_abs_ret_5d", "TM_Telephone_ret_1d", "XLB_Materials_zscore_60d", "AORD_AUS_zscore_60d", "XLY_Disc_vol_20d", "CPB_CampbellSoup_ret_20d", "BDX_Becton_Dickinson_ret_20d", "ASX_Australia_vol_20d", "NOC_Northrop_ret_20d", "EWY_Korea_ret_20d", "GD_GeneralDynamics_zscore_60d"], "is_new": true}, {"model_id": "new_h3_NORMAL_LightGBM_N15_t4", "algo": "LightGBM", "regime": "NORMAL", "horizon": 3, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "SLB_Schlumberger_ret_1d", "AMD_ret_5d", "FedFunds_zscore_60d", "DE_Deere_vol_20d", "AXP_Amex_ret_20d", "EFFR_ret_1d", "EFFR_vol_20d", "EWA_Australia_zscore_60d", "INTC_ret_1d", "US3M_Rate_vol_20d", "Retail_Sales_zscore_60d", "LLY_zscore_60d", "PCAR_PaccarInc_ret_5d"], "is_new": true}, {"model_id": "new_h3_NORMAL_LightGBM_N15_t5", "algo": "LightGBM", "regime": "NORMAL", "horizon": 3, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "SJM_JM_Smucker_ret_5d", "Retail_Sales_zscore_60d", "MSTR_Bitcoin3_ret_1d", "LLY_zscore_60d", "PAYX_Paychex_zscore_60d", "LOW_Lowes_ret_5d", "LOW_Lowes_ret_20d", "DHR_ret_1d", "IYM_BasicMaterials_ret_20d", "SLB_Schlumberger_ret_5d", "NFCI_ret_5d", "US1Y_Rate_ret_5d", "ITT_ITTInc_ret_5d"], "is_new": true}, {"model_id": "new_h3_NORMAL_LightGBM_N15_t6", "algo": "LightGBM", "regime": "NORMAL", "horizon": 3, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "MSTR_Bitcoin3_ret_5d", "GD_GeneralDynamics_zscore_60d", "SJM_JM_Smucker_ret_5d", "Retail_Sales_zscore_60d", "TED_Spread_vol_20d", "ENB_EnbridgeInc_ret_1d", "spx_momentum_3d", "M_Macys_vol_20d", "IBEX_Spain_ret_20d", "SLB_Schlumberger_ret_5d", "QQQ_vol_20d", "CTAS_Cintas_vol_20d", "EWM_Malaysia_ret_1d"], "is_new": true}, {"model_id": "new_h3_NORMAL_LightGBM_N15_t7", "algo": "LightGBM", "regime": "NORMAL", "horizon": 3, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "JNJ_ret_1d", "LMT_LockheedMartin_vol_20d", "EXC_Exelon_zscore_60d", "Michigan_Sentiment_ret_20d", "heston_var_ev_h5", "FedFunds_zscore_60d", "EWY_Korea_ret_20d", "EWQ_France_zscore_60d", "HD_ret_5d", "PAYX_Paychex_ret_20d", "XOM_ret_1d", "SLB_Schlumberger_ret_5d", "LMT_LockheedMartin_ret_1d"], "is_new": true}, {"model_id": "new_h3_NORMAL_LightGBM_N20_t0", "algo": "LightGBM", "regime": "NORMAL", "horizon": 3, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "PCAR_PaccarInc_ret_5d", "3M_vol_20d", "HD_ret_5d", "Core_PCE_zscore_60d", "IBEX_Spain_ret_20d", "MRK_Merck_zscore_60d", "ASX_Australia_vol_20d", "IYM_BasicMaterials_ret_20d", "vix_acceleration_1d", "MSTR_Bitcoin3_ret_5d", "EWJ_Japan_vol_20d", "EWC_Canada_zscore_60d", "SBUX_ret_5d", "AVB_AvalonBay_zscore_60d", "Nikkei_Japan_vol_20d", "XLB_Materials_zscore_60d", "IYR_US_REIT2_zscore_60d", "hmm_p_stress"], "is_new": true}, {"model_id": "new_h3_NORMAL_LightGBM_N20_t1", "algo": "LightGBM", "regime": "NORMAL", "horizon": 3, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "heston_var_ev_h7", "Core_PCE_zscore_60d", "EWC_Canada_zscore_60d", "TGT_Target_zscore_60d", "ORCL_zscore_60d", "LUV_SouthwestAir_ret_5d", "spx_abs_ret_max_5d", "XLV_Health_zscore_60d", "LMT_LockheedMartin_ret_1d", "IWM_SmallCap_vol_20d", "US3M_Rate_vol_20d", "HD_ret_1d", "EMR_Emerson_ret_20d", "EQIX_Equinix_ret_5d", "US5Y_Rate_ret_5d", "TED_Spread_vol_20d", "CPB_CampbellSoup_vol_20d", "vix_acceleration_1d"], "is_new": true}, {"model_id": "new_h3_NORMAL_LightGBM_N20_t2", "algo": "LightGBM", "regime": "NORMAL", "horizon": 3, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "Brent_Oil_FRED_ret_5d", "XLV_Health_zscore_60d", "vix_acceleration_1d", "PFE_ret_1d", "PG_ret_20d", "CLX_Clorox_vol_20d", "CCI_CrownCastle_vol_20d", "T10Y2Y_Spread_ret_5d", "3M_vol_20d", "CPB_CampbellSoup_zscore_60d", "MSTR_Bitcoin3_ret_1d", "PAYX_Paychex_ret_20d", "HUM_Humana_ret_5d", "EWA_Australia_zscore_60d", "MSTR_Bitcoin3_ret_20d", "BA_ret_1d", "FedFunds_zscore_60d", "TXN_vol_20d"], "is_new": true}, {"model_id": "new_h3_NORMAL_LightGBM_N20_t3", "algo": "LightGBM", "regime": "NORMAL", "horizon": 3, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "ORCL_zscore_60d", "CPB_CampbellSoup_ret_5d", "EXC_Exelon_zscore_60d", "EWM_Malaysia_ret_1d", "NOC_Northrop_ret_20d", "US3Y_Rate_ret_5d", "GE_ret_1d", "PPL_PPL_ret_1d", "EWY_Korea_zscore_60d", "XLB_Materials_zscore_60d", "EWG_Germany_vol_20d", "heston_ev_h3", "HangSeng_HK_vol_20d", "Retail_Sales_zscore_60d", "MS_MorganStanley_ret_1d", "SPY_zscore_60d", "TGT_Target_zscore_60d", "US7Y_Rate_ret_20d"], "is_new": true}, {"model_id": "new_h3_NORMAL_LightGBM_N20_t4", "algo": "LightGBM", "regime": "NORMAL", "horizon": 3, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "gjr_condvar_h1", "CMCSA_ret_1d", "TED_Spread_vol_20d", "vix_acceleration_1d", "SBUX_ret_5d", "PFE_ret_1d", "MSTR_Bitcoin3_ret_5d", "HangSeng_HK_vol_20d", "INTC_ret_1d", "BDX_Becton_Dickinson_ret_20d", "TM_Telephone_vol_20d", "HD_zscore_60d", "EWA_Australia_ret_1d", "CCI_CrownCastle_vol_20d", "XLY_Disc_vol_20d", "NOC_Northrop_ret_20d", "heston_ev_h3", "INTC_ret_5d"], "is_new": true}, {"model_id": "new_h3_NORMAL_LightGBM_N20_t5", "algo": "LightGBM", "regime": "NORMAL", "horizon": 3, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "heston_var_ev_h3", "AORD_AUS_zscore_60d", "ITT_ITTInc_ret_5d", "EWA_Australia_zscore_60d", "spx_vol_5d", "BTI_BritishAmerican_ret_20d", "DE_Deere_vol_20d", "NVDA_vol_20d", "AMT_AmericanTower_ret_1d", "MO_AltriaMG_ret_1d", "INTC_ret_5d", "SBUX_zscore_60d", "HangSeng_HK_ret_1d", "BLK_BlackRock_zscore_60d", "DHR_ret_1d", "XOM_ret_20d", "TGT_Target_zscore_60d", "HangSeng_HK_ret_5d"], "is_new": true}, {"model_id": "new_h3_NORMAL_LightGBM_N20_t6", "algo": "LightGBM", "regime": "NORMAL", "horizon": 3, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "US3Y_Rate_ret_5d", "ITT_ITTInc_ret_5d", "MRK_Merck_zscore_60d", "US3M_Rate_vol_20d", "CPB_CampbellSoup_ret_20d", "heston_var_ev_h3", "EOG_EOGResources_ret_5d", "NOC_Northrop_ret_20d", "SPY_zscore_60d", "CMCSA_ret_1d", "EWM_Malaysia_vol_20d", "ASX_Australia_ret_5d", "SCHW_Schwab_ret_5d", "EWQ_France_zscore_60d", "PAYX_Paychex_zscore_60d", "EWH_HongKong_ret_5d", "CPB_CampbellSoup_zscore_60d", "T10Y2Y_Spread_ret_5d"], "is_new": true}, {"model_id": "new_h3_NORMAL_LightGBM_N20_t7", "algo": "LightGBM", "regime": "NORMAL", "horizon": 3, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "CPB_CampbellSoup_zscore_60d", "EOG_EOGResources_ret_5d", "EWQ_France_ret_20d", "EXC_Exelon_zscore_60d", "IWM_SmallCap_vol_20d", "US7Y_Rate_ret_20d", "EOG_EOGResources_vol_20d", "EWG_Germany_ret_20d", "EQIX_Equinix_ret_5d", "MO_AltriaMG_ret_1d", "SPY_zscore_60d", "BDX_Becton_Dickinson_ret_20d", "EQR_Equity_ret_1d", "HD_ret_5d", "LLY_zscore_60d", "INTC_ret_5d", "SJM_JM_Smucker_ret_1d", "Brent_Oil_FRED_ret_5d"], "is_new": true}, {"model_id": "new_h3_NORMAL_LightGBM_N25_t0", "algo": "LightGBM", "regime": "NORMAL", "horizon": 3, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "GE_ret_1d", "EWH_HongKong_ret_5d", "IYR_US_REIT2_zscore_60d", "CPB_CampbellSoup_vol_20d", "HD_ret_20d", "VVIX_ret_20d", "DAX_Germany_zscore_60d", "SO_SouthernCo_ret_5d", "NFCI_ret_5d", "EWG_Germany_ret_20d", "DE_Deere_vol_20d", "SCHW_Schwab_ret_5d", "BDX_Becton_Dickinson_ret_20d", "EFFR_ret_1d", "EWG_Germany_vol_20d", "CPB_CampbellSoup_ret_5d", "PG_ret_20d", "EXC_Exelon_ret_1d", "ASX_Australia_ret_5d", "DE_Deere_ret_5d", "TED_Spread_vol_20d", "Brent_Oil_FRED_ret_5d", "AXP_Amex_ret_20d"], "is_new": true}, {"model_id": "new_h3_NORMAL_LightGBM_N25_t1", "algo": "LightGBM", "regime": "NORMAL", "horizon": 3, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWM_Malaysia_zscore_60d", "HUM_Humana_ret_5d", "GILD_Gilead_ret_20d", "EWH_HongKong_ret_5d", "spx_momentum_3d", "HD_ret_1d", "NOC_Northrop_ret_20d", "3M_ret_5d", "EQR_Equity_ret_1d", "MSTR_Bitcoin3_ret_1d", "spx_abs_ret_max_5d", "EWS_Singapore_ret_5d", "LUV_SouthwestAir_ret_5d", "MS_MorganStanley_zscore_60d", "hmm_p_stress", "Brent_Oil_FRED_ret_20d", "Michigan_Sentiment_ret_20d", "EQIX_Equinix_ret_5d", "VRP_ma5", "T_ret_1d", "SCHW_Schwab_ret_5d", "ASX_Australia_vol_20d", "EOG_EOGResources_ret_5d"], "is_new": true}, {"model_id": "new_h3_NORMAL_LightGBM_N25_t2", "algo": "LightGBM", "regime": "NORMAL", "horizon": 3, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "CLX_Clorox_vol_20d", "GD_GeneralDynamics_zscore_60d", "heston_var_ev_h3", "EWM_Malaysia_vol_20d", "LMT_LockheedMartin_ret_1d", "CPB_CampbellSoup_ret_20d", "QQQ_vol_20d", "DE_Deere_ret_5d", "EWC_Canada_zscore_60d", "MSTR_Bitcoin3_ret_20d", "SLB_Schlumberger_ret_1d", "LUV_SouthwestAir_ret_5d", "AXP_Amex_ret_20d", "FedFunds_zscore_60d", "IBEX_Spain_ret_20d", "TM_Telephone_ret_1d", "XOM_ret_20d", "EOG_EOGResources_ret_5d", "MS_MorganStanley_ret_1d", "BA_ret_1d", "DHR_vol_20d", "EWS_Singapore_ret_5d", "IYR_US_REIT2_zscore_60d"], "is_new": true}, {"model_id": "new_h3_NORMAL_LightGBM_N25_t3", "algo": "LightGBM", "regime": "NORMAL", "horizon": 3, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EXC_Exelon_ret_1d", "ASX_Australia_vol_20d", "INTC_ret_5d", "LOW_Lowes_ret_5d", "EWQ_France_ret_20d", "AXP_Amex_ret_20d", "Michigan_Sentiment_ret_20d", "Brent_Oil_FRED_ret_5d", "Core_CPI_zscore_60d", "EWM_Malaysia_ret_1d", "XLK_Tech_zscore_60d", "AMGN_Amgen_ret_1d", "3M_vol_20d", "GD_GeneralDynamics_zscore_60d", "LUV_SouthwestAir_ret_5d", "TED_Spread_zscore_60d", "XLB_Materials_zscore_60d", "DE_Deere_vol_20d", "QQQ_vol_20d", "XLV_Health_zscore_60d", "ENB_EnbridgeInc_ret_1d", "EWC_Canada_zscore_60d", "XLY_Disc_vol_20d"], "is_new": true}, {"model_id": "new_h3_NORMAL_LightGBM_N25_t4", "algo": "LightGBM", "regime": "NORMAL", "horizon": 3, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "NVDA_vol_20d", "US30Y_Rate_ret_20d", "EWM_Malaysia_ret_1d", "EWL_Switzerland_zscore_60d", "IYR_US_REIT2_zscore_60d", "LUV_SouthwestAir_ret_5d", "LOW_Lowes_ret_20d", "M_Macys_vol_20d", "EWQ_France_zscore_60d", "SLB_Schlumberger_ret_5d", "MO_AltriaMG_ret_1d", "VRP_ma5", "US1Y_Rate_ret_5d", "BTI_BritishAmerican_ret_20d", "TED_Spread_vol_20d", "EFFR_ret_1d", "XOM_ret_1d", "EXC_Exelon_zscore_60d", "EWH_HongKong_ret_5d", "US3M_Rate_vol_20d", "ORCL_zscore_60d", "HangSeng_HK_ret_1d", "TED_Spread_zscore_60d"], "is_new": true}, {"model_id": "new_h3_NORMAL_LightGBM_N25_t5", "algo": "LightGBM", "regime": "NORMAL", "horizon": 3, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "LOW_Lowes_ret_5d", "SBUX_zscore_60d", "EWH_HongKong_ret_5d", "EWG_Germany_ret_20d", "TED_Spread_vol_20d", "3M_ret_5d", "PFE_ret_1d", "XOM_ret_1d", "DAX_Germany_zscore_60d", "Michigan_Sentiment_ret_20d", "gjr_condvar_h1", "MS_MorganStanley_ret_1d", "hmm_p_stress", "US5Y_Rate_ret_5d", "HD_ret_5d", "AMD_ret_1d", "HangSeng_HK_ret_5d", "EWY_Korea_zscore_60d", "PCAR_PaccarInc_ret_5d", "MO_AltriaMG_ret_1d", "AMZN_ret_5d", "CPB_CampbellSoup_vol_20d", "WTI_Oil_FRED_zscore_60d"], "is_new": true}, {"model_id": "new_h3_NORMAL_LightGBM_N25_t6", "algo": "LightGBM", "regime": "NORMAL", "horizon": 3, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "MS_MorganStanley_ret_5d", "LLY_zscore_60d", "ITT_ITTInc_ret_5d", "SPY_zscore_60d", "VVIX_ret_20d", "XLK_Tech_zscore_60d", "heston_ev_h3", "ASX_Australia_vol_20d", "MS_MorganStanley_zscore_60d", "spx_vol_5d", "TGT_Target_zscore_60d", "heston_var_ev_h3", "VRP_ma5", "SJM_JM_Smucker_ret_1d", "DE_Deere_vol_20d", "EMR_Emerson_ret_20d", "EWG_Germany_ret_20d", "EOG_EOGResources_ret_5d", "HD_zscore_60d", "EQR_Equity_ret_1d", "AVB_AvalonBay_zscore_60d", "EWY_Korea_zscore_60d", "EWQ_France_zscore_60d"], "is_new": true}, {"model_id": "new_h3_NORMAL_LightGBM_N25_t7", "algo": "LightGBM", "regime": "NORMAL", "horizon": 3, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "BTI_BritishAmerican_ret_20d", "US3Y_Rate_ret_5d", "CPB_CampbellSoup_vol_20d", "ASX_Australia_vol_20d", "Nikkei_Japan_vol_20d", "US3M_Rate_vol_20d", "PPL_PPL_ret_1d", "CPB_CampbellSoup_ret_5d", "EWY_Korea_ret_20d", "MSTR_Bitcoin3_ret_20d", "EWQ_France_zscore_60d", "JNJ_ret_1d", "VRP_ma5", "CCI_CrownCastle_vol_20d", "CMCSA_ret_1d", "HangSeng_HK_ret_5d", "DIS_vol_20d", "AXP_Amex_vol_20d", "EFFR_ret_1d", "T_ret_1d", "MS_MorganStanley_zscore_60d", "CPB_CampbellSoup_zscore_60d", "DE_Deere_vol_20d"], "is_new": true}, {"model_id": "new_h3_NORMAL_LightGBM_N30_t0", "algo": "LightGBM", "regime": "NORMAL", "horizon": 3, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "Core_PCE_zscore_60d", "GE_ret_1d", "3M_ret_5d", "DE_Deere_ret_5d", "heston_ev_h3", "HD_ret_5d", "IYM_BasicMaterials_ret_20d", "HangSeng_HK_ret_1d", "DE_Deere_vol_20d", "DIS_vol_20d", "EXC_Exelon_zscore_60d", "VVIX_ret_20d", "EFFR_ret_1d", "EOG_EOGResources_ret_5d", "hmm_p_stress", "EWQ_France_zscore_60d", "AMT_AmericanTower_ret_1d", "DOW_Price_zscore_60d", "EWM_Malaysia_zscore_60d", "spx_momentum_3d", "BDX_Becton_Dickinson_ret_20d", "AMD_ret_5d", "AMZN_ret_5d", "CPB_CampbellSoup_ret_20d", "EWA_Australia_ret_1d", "XOM_ret_20d", "PAYX_Paychex_vol_20d", "TED_Spread_vol_20d"], "is_new": true}, {"model_id": "new_h3_NORMAL_LightGBM_N30_t1", "algo": "LightGBM", "regime": "NORMAL", "horizon": 3, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "spx_vol_5d", "EWQ_France_ret_20d", "IYR_US_REIT2_zscore_60d", "BLK_BlackRock_zscore_60d", "heston_var_ev_h5", "VOD_Vodafone_zscore_60d", "SBUX_zscore_60d", "US3M_Rate_zscore_60d", "CPB_CampbellSoup_ret_20d", "EWM_Malaysia_zscore_60d", "vix_mean_abs_ret_5d", "EWM_Malaysia_vol_20d", "EMR_Emerson_ret_20d", "XLB_Materials_zscore_60d", "ITT_ITTInc_ret_5d", "MSTR_Bitcoin3_ret_20d", "Core_CPI_zscore_60d", "SBUX_vol_20d", "NFCI_ret_5d", "ES_Evergy_ret_1d", "hmm_p_stress", "HD_ret_1d", "SPY_zscore_60d", "GILD_Gilead_ret_20d", "NEE_NextEra_ret_20d", "XLY_Disc_vol_20d", "MSTR_Bitcoin3_ret_1d", "HD_ret_5d"], "is_new": true}, {"model_id": "new_h3_NORMAL_LightGBM_N30_t2", "algo": "LightGBM", "regime": "NORMAL", "horizon": 3, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWL_Switzerland_vol_20d", "MSTR_Bitcoin3_ret_1d", "BTI_BritishAmerican_ret_5d", "GILD_Gilead_ret_20d", "3M_ret_5d", "US1Y_Rate_ret_20d", "EWA_Australia_ret_1d", "spx_abs_ret_max_5d", "PLD_Prologis_ret_5d", "AVB_AvalonBay_zscore_60d", "VVIX_ret_20d", "BA_ret_1d", "MS_MorganStanley_ret_5d", "EWS_Singapore_ret_5d", "EWQ_France_ret_20d", "vix_acceleration_1d", "BDX_Becton_Dickinson_ret_20d", "DAX_Germany_zscore_60d", "SBUX_vol_20d", "heston_var_ev_h7", "DAX_Germany_vol_20d", "IYR_US_REIT2_zscore_60d", "NWL_Newell_ret_20d", "PPL_PPL_ret_1d", "EMR_Emerson_ret_20d", "NVDA_vol_20d", "spx_vol_5d", "SBUX_zscore_60d"], "is_new": true}, {"model_id": "new_h3_NORMAL_LightGBM_N30_t3", "algo": "LightGBM", "regime": "NORMAL", "horizon": 3, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "VOD_Vodafone_zscore_60d", "DIS_vol_20d", "HD_zscore_60d", "ITT_ITTInc_ret_5d", "WTI_Oil_FRED_zscore_60d", "TXN_vol_20d", "US3Y_Rate_ret_5d", "PPL_PPL_ret_1d", "PAYX_Paychex_zscore_60d", "PCAR_PaccarInc_ret_5d", "EWC_Canada_zscore_60d", "hmm_p_stress", "EQR_Equity_ret_1d", "HangSeng_HK_ret_1d", "EWA_Australia_zscore_60d", "SJM_JM_Smucker_ret_5d", "EFFR_ret_1d", "BTI_BritishAmerican_ret_20d", "HD_ret_1d", "Nikkei_Japan_zscore_60d", "ENB_EnbridgeInc_ret_1d", "EFFR_vol_20d", "VRP_ma5", "ES_Evergy_ret_1d", "EMR_Emerson_ret_20d", "TED_Spread_zscore_60d", "EWM_Malaysia_ret_1d", "gjr_condvar_h1"], "is_new": true}, {"model_id": "new_h3_NORMAL_LightGBM_N30_t4", "algo": "LightGBM", "regime": "NORMAL", "horizon": 3, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EQIX_Equinix_ret_5d", "EMR_Emerson_ret_20d", "EWL_Switzerland_vol_20d", "Industrial_Production_zscore_60d", "WTI_Oil_FRED_zscore_60d", "AORD_AUS_zscore_60d", "MSTR_Bitcoin3_ret_5d", "DIS_vol_20d", "SBUX_vol_20d", "hmm_p_stress", "US6M_Rate_ret_20d", "HD_zscore_60d", "AVB_AvalonBay_zscore_60d", "CPB_CampbellSoup_ret_20d", "HD_ret_1d", "HangSeng_HK_ret_1d", "Brent_Oil_FRED_ret_5d", "EOG_EOGResources_ret_5d", "IBEX_Spain_ret_20d", "spx_momentum_3d", "vix_acceleration_1d", "ES_Evergy_ret_1d", "EFFR_ret_1d", "EWY_Korea_zscore_60d", "MO_AltriaMG_ret_1d", "MS_MorganStanley_ret_1d", "PAYX_Paychex_ret_20d", "AMZN_ret_5d"], "is_new": true}, {"model_id": "new_h3_NORMAL_LightGBM_N30_t5", "algo": "LightGBM", "regime": "NORMAL", "horizon": 3, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "ASX_Australia_ret_5d", "HD_ret_1d", "NOC_Northrop_ret_20d", "EFFR_vol_20d", "XLV_Health_zscore_60d", "EWA_Australia_ret_1d", "EWC_Canada_zscore_60d", "heston_var_ev_h5", "PFE_ret_1d", "vix_acceleration_1d", "CTAS_Cintas_vol_20d", "LMT_LockheedMartin_vol_20d", "heston_var_ev_h7", "JNJ_ret_1d", "US1Y_Rate_ret_20d", "EWM_Malaysia_zscore_60d", "EWH_HongKong_ret_5d", "T10Y2Y_Spread_ret_5d", "PAYX_Paychex_vol_20d", "SLB_Schlumberger_ret_5d", "US7Y_Rate_ret_20d", "LUV_SouthwestAir_ret_5d", "CCI_CrownCastle_vol_20d", "AMT_AmericanTower_ret_1d", "XLY_Disc_vol_20d", "EWS_Singapore_ret_5d", "vix_mean_abs_ret_5d", "GD_GeneralDynamics_zscore_60d"], "is_new": true}, {"model_id": "new_h3_NORMAL_LightGBM_N30_t6", "algo": "LightGBM", "regime": "NORMAL", "horizon": 3, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "BLK_BlackRock_zscore_60d", "SLB_Schlumberger_ret_5d", "ES_Evergy_ret_1d", "LLY_zscore_60d", "Retail_Sales_zscore_60d", "DE_Deere_ret_5d", "EWM_Malaysia_ret_1d", "EQR_Equity_ret_1d", "EWA_Australia_ret_1d", "AMD_ret_5d", "3M_vol_20d", "SBUX_ret_5d", "gjr_condvar_h1", "US30Y_Rate_ret_20d", "NVDA_vol_20d", "BTI_BritishAmerican_ret_5d", "US1Y_Rate_ret_5d", "ORCL_vol_20d", "HangSeng_HK_ret_5d", "SJM_JM_Smucker_ret_5d", "HangSeng_HK_ret_1d", "MS_MorganStanley_ret_5d", "PCAR_PaccarInc_ret_5d", "HD_ret_20d", "PLD_Prologis_ret_5d", "LMT_LockheedMartin_ret_1d", "EOG_EOGResources_ret_5d", "CPB_CampbellSoup_ret_5d"], "is_new": true}, {"model_id": "new_h3_NORMAL_LightGBM_N30_t7", "algo": "LightGBM", "regime": "NORMAL", "horizon": 3, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "LOW_Lowes_ret_5d", "AVB_AvalonBay_zscore_60d", "US1Y_Rate_ret_5d", "heston_var_ev_h3", "MSTR_Bitcoin3_ret_20d", "US3Y_Rate_ret_5d", "DE_Deere_vol_20d", "DHR_ret_1d", "PFE_ret_1d", "3M_ret_5d", "EOG_EOGResources_vol_20d", "IYR_US_REIT2_zscore_60d", "DE_Deere_ret_5d", "MO_AltriaMG_ret_1d", "FedFunds_zscore_60d", "CPB_CampbellSoup_vol_20d", "CTAS_Cintas_vol_20d", "BLK_BlackRock_zscore_60d", "XLY_Disc_vol_20d", "LMT_LockheedMartin_vol_20d", "NFCI_ret_5d", "EQIX_Equinix_ret_5d", "HangSeng_HK_ret_5d", "Brent_Oil_FRED_ret_20d", "DHR_vol_20d", "NWL_Newell_ret_20d", "NEE_NextEra_ret_20d", "PAYX_Paychex_vol_20d"], "is_new": true}, {"model_id": "new_h3_NORMAL_GradientBoosting_N5_t0", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 3, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "INTC_ret_1d", "US3M_Rate_zscore_60d", "M_Macys_vol_20d"], "is_new": true}, {"model_id": "new_h3_NORMAL_GradientBoosting_N5_t1", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 3, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWG_Germany_ret_20d", "NWL_Newell_ret_20d", "GD_GeneralDynamics_zscore_60d"], "is_new": true}, {"model_id": "new_h3_NORMAL_GradientBoosting_N5_t2", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 3, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EFFR_ret_1d", "AMZN_ret_5d", "Nikkei_Japan_zscore_60d"], "is_new": true}, {"model_id": "new_h3_NORMAL_GradientBoosting_N5_t3", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 3, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "NEE_NextEra_ret_20d", "heston_ev_h3", "BTI_BritishAmerican_ret_5d"], "is_new": true}, {"model_id": "new_h3_NORMAL_GradientBoosting_N5_t4", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 3, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "vix_acceleration_1d", "EWH_HongKong_ret_5d", "Retail_Sales_zscore_60d"], "is_new": true}, {"model_id": "new_h3_NORMAL_GradientBoosting_N5_t5", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 3, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWA_Australia_ret_1d", "Core_PCE_zscore_60d", "AXP_Amex_ret_20d"], "is_new": true}, {"model_id": "new_h3_NORMAL_GradientBoosting_N5_t6", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 3, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "TGT_Target_zscore_60d", "EWY_Korea_ret_20d", "hmm_p_stress"], "is_new": true}, {"model_id": "new_h3_NORMAL_GradientBoosting_N5_t7", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 3, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "ES_Evergy_ret_1d", "EMR_Emerson_ret_20d", "LUV_SouthwestAir_ret_5d"], "is_new": true}, {"model_id": "new_h3_NORMAL_GradientBoosting_N8_t0", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 3, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "HD_ret_20d", "ORCL_zscore_60d", "T_ret_1d", "PG_ret_20d", "BLK_BlackRock_zscore_60d", "ORCL_vol_20d"], "is_new": true}, {"model_id": "new_h3_NORMAL_GradientBoosting_N8_t1", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 3, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EMR_Emerson_ret_20d", "SLB_Schlumberger_ret_1d", "NFCI_ret_5d", "XOM_ret_20d", "EWM_Malaysia_zscore_60d", "ASX_Australia_vol_20d"], "is_new": true}, {"model_id": "new_h3_NORMAL_GradientBoosting_N8_t2", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 3, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "AVB_AvalonBay_zscore_60d", "BTI_BritishAmerican_ret_5d", "ASX_Australia_vol_20d", "CPB_CampbellSoup_zscore_60d", "DE_Deere_vol_20d", "HUM_Humana_ret_5d"], "is_new": true}, {"model_id": "new_h3_NORMAL_GradientBoosting_N8_t3", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 3, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "XLF_Fin_vol_20d", "heston_var_ev_h3", "HangSeng_HK_ret_1d", "DAX_Germany_zscore_60d", "CPB_CampbellSoup_vol_20d", "CTAS_Cintas_vol_20d"], "is_new": true}, {"model_id": "new_h3_NORMAL_GradientBoosting_N8_t4", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 3, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "AMZN_ret_5d", "HD_ret_1d", "EXC_Exelon_ret_1d", "US7Y_Rate_ret_20d", "BTI_BritishAmerican_ret_5d", "BA_ret_1d"], "is_new": true}, {"model_id": "new_h3_NORMAL_GradientBoosting_N8_t5", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 3, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWA_Australia_zscore_60d", "Brent_Oil_FRED_ret_20d", "HUM_Humana_ret_5d", "US3M_Rate_zscore_60d", "EOG_EOGResources_vol_20d", "AORD_AUS_zscore_60d"], "is_new": true}, {"model_id": "new_h3_NORMAL_GradientBoosting_N8_t6", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 3, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "HD_zscore_60d", "BDX_Becton_Dickinson_ret_20d", "FedFunds_zscore_60d", "CI_Cigna_vol_20d", "DE_Deere_ret_5d", "Brent_Oil_FRED_ret_20d"], "is_new": true}, {"model_id": "new_h3_NORMAL_GradientBoosting_N8_t7", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 3, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWG_Germany_vol_20d", "Brent_Oil_FRED_ret_5d", "AORD_AUS_zscore_60d", "LMT_LockheedMartin_ret_1d", "MO_AltriaMG_ret_1d", "AMD_ret_1d"], "is_new": true}, {"model_id": "new_h3_NORMAL_GradientBoosting_N10_t0", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 3, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "T_ret_1d", "INTC_ret_5d", "ASX_Australia_vol_20d", "MS_MorganStanley_ret_1d", "NOC_Northrop_ret_20d", "AMD_ret_5d", "Nikkei_Japan_zscore_60d", "PAYX_Paychex_ret_20d"], "is_new": true}, {"model_id": "new_h3_NORMAL_GradientBoosting_N10_t1", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 3, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "FedFunds_zscore_60d", "T10Y2Y_Spread_ret_5d", "EWM_Malaysia_vol_20d", "LLY_zscore_60d", "Brent_Oil_FRED_ret_20d", "IYR_US_REIT2_zscore_60d", "Michigan_Sentiment_ret_20d", "AMT_AmericanTower_ret_1d"], "is_new": true}, {"model_id": "new_h3_NORMAL_GradientBoosting_N10_t2", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 3, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "DE_Deere_ret_5d", "PFE_ret_1d", "QQQ_vol_20d", "BA_ret_1d", "EFFR_ret_1d", "BTI_BritishAmerican_ret_20d", "JNJ_ret_1d", "EWM_Malaysia_vol_20d"], "is_new": true}, {"model_id": "new_h3_NORMAL_GradientBoosting_N10_t3", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 3, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "IYR_US_REIT2_zscore_60d", "DAX_Germany_vol_20d", "Core_PCE_zscore_60d", "CCI_CrownCastle_vol_20d", "ORCL_vol_20d", "EOG_EOGResources_vol_20d", "PPL_PPL_ret_1d", "CPB_CampbellSoup_ret_20d"], "is_new": true}, {"model_id": "new_h3_NORMAL_GradientBoosting_N10_t4", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 3, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "FedFunds_zscore_60d", "CTAS_Cintas_vol_20d", "ASX_Australia_vol_20d", "MS_MorganStanley_ret_5d", "CPB_CampbellSoup_vol_20d", "heston_var_ev_h7", "NVDA_vol_20d", "IWM_SmallCap_vol_20d"], "is_new": true}, {"model_id": "new_h3_NORMAL_GradientBoosting_N10_t5", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 3, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "INTC_ret_5d", "LMT_LockheedMartin_ret_1d", "EWL_Switzerland_vol_20d", "DAX_Germany_vol_20d", "ASX_Australia_vol_20d", "SJM_JM_Smucker_ret_1d", "EWG_Germany_ret_20d", "Brent_Oil_FRED_ret_5d"], "is_new": true}, {"model_id": "new_h3_NORMAL_GradientBoosting_N10_t6", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 3, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "DIS_vol_20d", "ITT_ITTInc_ret_5d", "ASX_Australia_ret_5d", "TXN_vol_20d", "heston_var_ev_h7", "HangSeng_HK_vol_20d", "EQIX_Equinix_ret_5d", "HangSeng_HK_ret_1d"], "is_new": true}, {"model_id": "new_h3_NORMAL_GradientBoosting_N10_t7", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 3, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "VRP_ma5", "DAX_Germany_zscore_60d", "INTC_ret_5d", "EWM_Malaysia_vol_20d", "EWQ_France_ret_20d", "NEE_NextEra_ret_20d", "ASX_Australia_vol_20d", "CPB_CampbellSoup_ret_5d"], "is_new": true}, {"model_id": "new_h3_NORMAL_GradientBoosting_N12_t0", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 3, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "VRP_ma5", "CPB_CampbellSoup_ret_5d", "GD_GeneralDynamics_zscore_60d", "BLK_BlackRock_zscore_60d", "AMZN_ret_5d", "VVIX_ret_20d", "XLK_Tech_zscore_60d", "SLB_Schlumberger_ret_1d", "Core_CPI_zscore_60d", "ENB_EnbridgeInc_ret_1d"], "is_new": true}, {"model_id": "new_h3_NORMAL_GradientBoosting_N12_t1", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 3, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "spx_vol_5d", "heston_var_ev_h5", "ES_Evergy_ret_1d", "SPY_zscore_60d", "T10Y2Y_Spread_ret_5d", "PAYX_Paychex_vol_20d", "HangSeng_HK_vol_20d", "Brent_Oil_FRED_ret_5d", "IYR_US_REIT2_zscore_60d", "VRP_ma5"], "is_new": true}, {"model_id": "new_h3_NORMAL_GradientBoosting_N12_t2", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 3, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "heston_ev_h3", "EWA_Australia_zscore_60d", "HangSeng_HK_ret_1d", "IYM_BasicMaterials_ret_20d", "AXP_Amex_ret_20d", "DOW_Price_zscore_60d", "Michigan_Sentiment_ret_20d", "Nikkei_Japan_vol_20d", "EXC_Exelon_zscore_60d", "heston_var_ev_h7"], "is_new": true}, {"model_id": "new_h3_NORMAL_GradientBoosting_N12_t3", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 3, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "XOM_ret_20d", "VVIX_ret_20d", "MS_MorganStanley_ret_5d", "ORCL_vol_20d", "HD_ret_1d", "T_ret_1d", "BTI_BritishAmerican_ret_20d", "CI_Cigna_vol_20d", "ASX_Australia_vol_20d", "IYR_US_REIT2_zscore_60d"], "is_new": true}, {"model_id": "new_h3_NORMAL_GradientBoosting_N12_t4", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 3, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "DIS_vol_20d", "GE_ret_1d", "EWA_Australia_ret_1d", "AMGN_Amgen_ret_1d", "Nikkei_Japan_vol_20d", "SCHW_Schwab_ret_5d", "PPL_PPL_ret_1d", "SJM_JM_Smucker_ret_1d", "XLY_Disc_vol_20d", "SPY_zscore_60d"], "is_new": true}, {"model_id": "new_h3_NORMAL_GradientBoosting_N12_t5", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 3, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "SLB_Schlumberger_ret_1d", "BTI_BritishAmerican_ret_5d", "AMD_ret_1d", "SCHW_Schwab_ret_5d", "3M_ret_5d", "SLB_Schlumberger_ret_5d", "DE_Deere_vol_20d", "EWY_Korea_zscore_60d", "BA_ret_1d", "SJM_JM_Smucker_ret_5d"], "is_new": true}, {"model_id": "new_h3_NORMAL_GradientBoosting_N12_t6", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 3, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWL_Switzerland_zscore_60d", "EWA_Australia_ret_1d", "CPB_CampbellSoup_ret_20d", "LMT_LockheedMartin_vol_20d", "PCAR_PaccarInc_ret_5d", "US5Y_Rate_ret_5d", "EQIX_Equinix_ret_5d", "BDX_Becton_Dickinson_ret_20d", "US30Y_Rate_ret_20d", "EQR_Equity_ret_1d"], "is_new": true}, {"model_id": "new_h3_NORMAL_GradientBoosting_N12_t7", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 3, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "CI_Cigna_vol_20d", "MS_MorganStanley_ret_1d", "heston_var_ev_h5", "EMR_Emerson_ret_20d", "AMGN_Amgen_ret_1d", "PAYX_Paychex_vol_20d", "DOW_Price_zscore_60d", "VOD_Vodafone_zscore_60d", "IWM_SmallCap_vol_20d", "Core_CPI_zscore_60d"], "is_new": true}, {"model_id": "new_h3_NORMAL_GradientBoosting_N15_t0", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 3, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "BDX_Becton_Dickinson_ret_20d", "US30Y_Rate_ret_20d", "NVDA_vol_20d", "VOD_Vodafone_zscore_60d", "NWL_Newell_ret_20d", "CTAS_Cintas_vol_20d", "CLX_Clorox_vol_20d", "AMT_AmericanTower_ret_1d", "US1Y_Rate_ret_20d", "GE_ret_1d", "3M_ret_5d", "EWQ_France_zscore_60d", "XOM_ret_1d"], "is_new": true}, {"model_id": "new_h3_NORMAL_GradientBoosting_N15_t1", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 3, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "SBUX_ret_5d", "EWL_Switzerland_vol_20d", "NFCI_ret_5d", "AMT_AmericanTower_ret_1d", "TED_Spread_zscore_60d", "NOC_Northrop_ret_20d", "EWM_Malaysia_vol_20d", "EWG_Germany_vol_20d", "US7Y_Rate_ret_20d", "VVIX_ret_20d", "heston_var_ev_h7", "LLY_zscore_60d", "AXP_Amex_ret_20d"], "is_new": true}, {"model_id": "new_h3_NORMAL_GradientBoosting_N15_t2", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 3, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "Nikkei_Japan_vol_20d", "AMD_ret_1d", "JNJ_ret_1d", "LMT_LockheedMartin_vol_20d", "AMT_AmericanTower_ret_1d", "TED_Spread_vol_20d", "heston_var_ev_h7", "AMZN_ret_5d", "hmm_p_stress", "ASX_Australia_vol_20d", "Industrial_Production_zscore_60d", "AXP_Amex_vol_20d", "XLF_Fin_vol_20d"], "is_new": true}, {"model_id": "new_h3_NORMAL_GradientBoosting_N15_t3", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 3, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "XOM_ret_20d", "XOM_ret_1d", "ES_Evergy_ret_1d", "LOW_Lowes_ret_5d", "AXP_Amex_vol_20d", "VRP_ma5", "SLB_Schlumberger_ret_1d", "ASX_Australia_vol_20d", "LMT_LockheedMartin_ret_1d", "MO_AltriaMG_ret_1d", "Nikkei_Japan_zscore_60d", "HUM_Humana_ret_5d", "IBEX_Spain_ret_20d"], "is_new": true}, {"model_id": "new_h3_NORMAL_GradientBoosting_N15_t4", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 3, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EXC_Exelon_zscore_60d", "EWJ_Japan_vol_20d", "NOC_Northrop_ret_20d", "BTI_BritishAmerican_ret_20d", "MS_MorganStanley_ret_5d", "AXP_Amex_vol_20d", "MRK_Merck_zscore_60d", "AMD_ret_5d", "EQR_Equity_ret_1d", "VVIX_ret_20d", "heston_ev_h3", "US7Y_Rate_ret_20d", "XLK_Tech_zscore_60d"], "is_new": true}, {"model_id": "new_h3_NORMAL_GradientBoosting_N15_t5", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 3, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "US6M_Rate_ret_20d", "Core_PCE_zscore_60d", "gjr_condvar_h1", "SLB_Schlumberger_ret_1d", "CTAS_Cintas_vol_20d", "IBEX_Spain_ret_20d", "XLK_Tech_zscore_60d", "TM_Telephone_ret_1d", "EWH_HongKong_ret_5d", "NFCI_ret_5d", "SCHW_Schwab_ret_5d", "T10Y2Y_Spread_ret_5d", "FedFunds_zscore_60d"], "is_new": true}, {"model_id": "new_h3_NORMAL_GradientBoosting_N15_t6", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 3, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "CMCSA_ret_1d", "PAYX_Paychex_zscore_60d", "PAYX_Paychex_ret_20d", "HD_ret_1d", "US3M_Rate_vol_20d", "Core_PCE_zscore_60d", "Retail_Sales_zscore_60d", "ORCL_vol_20d", "ORCL_zscore_60d", "IYR_US_REIT2_zscore_60d", "MRK_Merck_zscore_60d", "AMT_AmericanTower_ret_1d", "vix_acceleration_1d"], "is_new": true}, {"model_id": "new_h3_NORMAL_GradientBoosting_N15_t7", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 3, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "Industrial_Production_zscore_60d", "heston_var_ev_h5", "EWC_Canada_zscore_60d", "CPB_CampbellSoup_vol_20d", "IYR_US_REIT2_zscore_60d", "PAYX_Paychex_vol_20d", "LMT_LockheedMartin_vol_20d", "heston_var_ev_h7", "vix_acceleration_1d", "DE_Deere_ret_5d", "EQIX_Equinix_ret_5d", "XLY_Disc_vol_20d", "ITT_ITTInc_ret_5d"], "is_new": true}, {"model_id": "new_h3_NORMAL_GradientBoosting_N20_t0", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 3, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWC_Canada_zscore_60d", "US7Y_Rate_ret_20d", "BLK_BlackRock_zscore_60d", "EWJ_Japan_vol_20d", "NWL_Newell_ret_20d", "SCHW_Schwab_ret_5d", "HD_zscore_60d", "US1Y_Rate_ret_5d", "Nikkei_Japan_vol_20d", "PAYX_Paychex_ret_20d", "LOW_Lowes_ret_5d", "MS_MorganStanley_zscore_60d", "PPL_PPL_ret_1d", "MO_AltriaMG_ret_1d", "EWG_Germany_vol_20d", "IWM_SmallCap_vol_20d", "Brent_Oil_FRED_ret_5d", "INTC_ret_1d"], "is_new": true}, {"model_id": "new_h3_NORMAL_GradientBoosting_N20_t1", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 3, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWG_Germany_ret_20d", "WTI_Oil_FRED_zscore_60d", "DE_Deere_vol_20d", "SJM_JM_Smucker_ret_1d", "3M_ret_5d", "SPY_zscore_60d", "CPB_CampbellSoup_vol_20d", "EOG_EOGResources_ret_5d", "BDX_Becton_Dickinson_ret_20d", "PFE_ret_1d", "TXN_vol_20d", "AXP_Amex_ret_20d", "AMZN_ret_5d", "EWH_HongKong_ret_5d", "SCHW_Schwab_ret_5d", "TM_Telephone_vol_20d", "CLX_Clorox_vol_20d", "Core_CPI_zscore_60d"], "is_new": true}, {"model_id": "new_h3_NORMAL_GradientBoosting_N20_t2", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 3, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "SJM_JM_Smucker_ret_5d", "DOW_Price_zscore_60d", "EWQ_France_zscore_60d", "EXC_Exelon_ret_1d", "CPB_CampbellSoup_ret_20d", "Michigan_Sentiment_ret_20d", "AMD_ret_1d", "VVIX_ret_20d", "XLV_Health_zscore_60d", "AMZN_ret_5d", "US5Y_Rate_ret_5d", "PG_ret_20d", "LLY_zscore_60d", "SBUX_zscore_60d", "T10Y2Y_Spread_ret_5d", "TGT_Target_zscore_60d", "LMT_LockheedMartin_vol_20d", "Core_CPI_zscore_60d"], "is_new": true}, {"model_id": "new_h3_NORMAL_GradientBoosting_N20_t3", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 3, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "SJM_JM_Smucker_ret_1d", "MS_MorganStanley_zscore_60d", "EXC_Exelon_zscore_60d", "IBEX_Spain_ret_20d", "GD_GeneralDynamics_zscore_60d", "WTI_Oil_FRED_zscore_60d", "AXP_Amex_vol_20d", "HD_ret_1d", "EWM_Malaysia_ret_1d", "VRP_ma5", "HD_ret_5d", "vix_mean_abs_ret_5d", "M_Macys_vol_20d", "Brent_Oil_FRED_ret_5d", "ORCL_vol_20d", "heston_var_ev_h7", "DHR_vol_20d", "DAX_Germany_zscore_60d"], "is_new": true}, {"model_id": "new_h3_NORMAL_GradientBoosting_N20_t4", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 3, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "DE_Deere_ret_5d", "Nikkei_Japan_zscore_60d", "CPB_CampbellSoup_vol_20d", "AMGN_Amgen_ret_1d", "JNJ_ret_1d", "GD_GeneralDynamics_zscore_60d", "heston_var_ev_h7", "ASX_Australia_vol_20d", "EOG_EOGResources_ret_5d", "MSTR_Bitcoin3_ret_20d", "EQR_Equity_ret_1d", "T10Y2Y_Spread_ret_5d", "hmm_p_stress", "HangSeng_HK_ret_5d", "NFCI_ret_5d", "LOW_Lowes_ret_20d", "IYM_BasicMaterials_ret_20d", "US7Y_Rate_ret_20d"], "is_new": true}, {"model_id": "new_h3_NORMAL_GradientBoosting_N20_t5", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 3, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "TM_Telephone_vol_20d", "SLB_Schlumberger_ret_1d", "M_Macys_vol_20d", "EWY_Korea_zscore_60d", "XOM_ret_1d", "US5Y_Rate_ret_5d", "EWL_Switzerland_vol_20d", "EWA_Australia_zscore_60d", "vix_mean_abs_ret_5d", "US7Y_Rate_ret_20d", "PAYX_Paychex_zscore_60d", "GILD_Gilead_ret_20d", "LMT_LockheedMartin_vol_20d", "EWA_Australia_ret_1d", "XLF_Fin_vol_20d", "BTI_BritishAmerican_ret_20d", "TGT_Target_zscore_60d", "AVB_AvalonBay_zscore_60d"], "is_new": true}, {"model_id": "new_h3_NORMAL_GradientBoosting_N20_t6", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 3, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "NFCI_ret_5d", "CI_Cigna_vol_20d", "heston_var_ev_h3", "HangSeng_HK_ret_5d", "EXC_Exelon_ret_1d", "EWM_Malaysia_ret_1d", "vix_mean_abs_ret_5d", "ORCL_zscore_60d", "EWM_Malaysia_zscore_60d", "MSTR_Bitcoin3_ret_1d", "heston_var_ev_h5", "heston_var_ev_h7", "ASX_Australia_vol_20d", "XLY_Disc_vol_20d", "BLK_BlackRock_zscore_60d", "SO_SouthernCo_ret_5d", "IYM_BasicMaterials_ret_20d", "Industrial_Production_zscore_60d"], "is_new": true}, {"model_id": "new_h3_NORMAL_GradientBoosting_N20_t7", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 3, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "MS_MorganStanley_ret_5d", "EWL_Switzerland_vol_20d", "CPB_CampbellSoup_vol_20d", "TED_Spread_zscore_60d", "spx_abs_ret_max_5d", "EXC_Exelon_ret_1d", "EQR_Equity_ret_1d", "US3Y_Rate_ret_5d", "AMGN_Amgen_ret_1d", "AMD_ret_5d", "EFFR_vol_20d", "HD_ret_5d", "VRP_ma5", "ENB_EnbridgeInc_ret_1d", "heston_var_ev_h5", "SBUX_vol_20d", "TM_Telephone_ret_1d", "JNJ_ret_1d"], "is_new": true}, {"model_id": "new_h3_NORMAL_GradientBoosting_N25_t0", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 3, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "spx_momentum_3d", "CCI_CrownCastle_vol_20d", "INTC_ret_5d", "BTI_BritishAmerican_ret_5d", "EFFR_vol_20d", "CTAS_Cintas_vol_20d", "Nikkei_Japan_vol_20d", "SBUX_zscore_60d", "Michigan_Sentiment_ret_20d", "3M_vol_20d", "T10Y2Y_Spread_ret_5d", "XLF_Fin_vol_20d", "NEE_NextEra_ret_20d", "AXP_Amex_ret_20d", "CLX_Clorox_vol_20d", "AMT_AmericanTower_ret_1d", "MSTR_Bitcoin3_ret_1d", "heston_var_ev_h7", "spx_vol_5d", "SLB_Schlumberger_ret_1d", "PAYX_Paychex_vol_20d", "SJM_JM_Smucker_ret_5d", "spx_abs_ret_max_5d"], "is_new": true}, {"model_id": "new_h3_NORMAL_GradientBoosting_N25_t1", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 3, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "NVDA_vol_20d", "LOW_Lowes_ret_5d", "US1Y_Rate_ret_20d", "DE_Deere_ret_5d", "US5Y_Rate_ret_5d", "EWL_Switzerland_vol_20d", "US1Y_Rate_ret_5d", "IYM_BasicMaterials_ret_20d", "EWY_Korea_zscore_60d", "AVB_AvalonBay_zscore_60d", "XOM_ret_20d", "HangSeng_HK_ret_1d", "HUM_Humana_ret_5d", "Industrial_Production_zscore_60d", "AMGN_Amgen_ret_1d", "EWM_Malaysia_zscore_60d", "spx_abs_ret_max_5d", "T_ret_1d", "M_Macys_vol_20d", "EWA_Australia_zscore_60d", "TED_Spread_vol_20d", "EWC_Canada_zscore_60d", "heston_var_ev_h3"], "is_new": true}, {"model_id": "new_h3_NORMAL_GradientBoosting_N25_t2", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 3, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EXC_Exelon_ret_1d", "HUM_Humana_ret_5d", "XLF_Fin_vol_20d", "TED_Spread_zscore_60d", "MS_MorganStanley_ret_5d", "AXP_Amex_ret_20d", "DHR_ret_1d", "XLB_Materials_zscore_60d", "Michigan_Sentiment_ret_20d", "IYM_BasicMaterials_ret_20d", "EWM_Malaysia_ret_1d", "EQR_Equity_ret_1d", "US1Y_Rate_ret_5d", "gjr_condvar_h1", "Brent_Oil_FRED_ret_5d", "LLY_zscore_60d", "EWG_Germany_ret_20d", "AMT_AmericanTower_ret_1d", "Retail_Sales_zscore_60d", "AMGN_Amgen_ret_1d", "PLD_Prologis_ret_5d", "CPB_CampbellSoup_vol_20d", "AVB_AvalonBay_zscore_60d"], "is_new": true}, {"model_id": "new_h3_NORMAL_GradientBoosting_N25_t3", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 3, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "Brent_Oil_FRED_ret_5d", "LOW_Lowes_ret_20d", "BLK_BlackRock_zscore_60d", "SO_SouthernCo_ret_5d", "heston_var_ev_h3", "EWM_Malaysia_ret_1d", "EWQ_France_zscore_60d", "GILD_Gilead_ret_20d", "INTC_ret_5d", "Brent_Oil_FRED_ret_20d", "EQIX_Equinix_ret_5d", "XLF_Fin_vol_20d", "TED_Spread_zscore_60d", "DOW_Price_zscore_60d", "XLB_Materials_zscore_60d", "EWY_Korea_zscore_60d", "LLY_zscore_60d", "GE_ret_1d", "CLX_Clorox_vol_20d", "SLB_Schlumberger_ret_5d", "EWH_HongKong_ret_5d", "EWG_Germany_ret_20d", "SJM_JM_Smucker_ret_1d"], "is_new": true}, {"model_id": "new_h3_NORMAL_GradientBoosting_N25_t4", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 3, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "HangSeng_HK_vol_20d", "DIS_vol_20d", "LMT_LockheedMartin_ret_1d", "AMZN_ret_5d", "PFE_ret_1d", "vix_mean_abs_ret_5d", "EWL_Switzerland_zscore_60d", "TED_Spread_vol_20d", "DE_Deere_vol_20d", "AXP_Amex_ret_20d", "EWG_Germany_ret_20d", "VVIX_ret_20d", "LOW_Lowes_ret_20d", "BLK_BlackRock_zscore_60d", "BDX_Becton_Dickinson_ret_20d", "PAYX_Paychex_ret_20d", "SLB_Schlumberger_ret_5d", "Nikkei_Japan_vol_20d", "PAYX_Paychex_vol_20d", "WTI_Oil_FRED_zscore_60d", "IYR_US_REIT2_zscore_60d", "VRP_ma5", "3M_ret_5d"], "is_new": true}, {"model_id": "new_h3_NORMAL_GradientBoosting_N25_t5", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 3, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "SBUX_ret_5d", "heston_var_ev_h3", "gjr_condvar_h1", "MSTR_Bitcoin3_ret_5d", "BLK_BlackRock_zscore_60d", "hmm_p_stress", "EWL_Switzerland_zscore_60d", "MS_MorganStanley_ret_5d", "PAYX_Paychex_zscore_60d", "EWH_HongKong_ret_5d", "EWL_Switzerland_vol_20d", "MS_MorganStanley_ret_1d", "BTI_BritishAmerican_ret_5d", "GD_GeneralDynamics_zscore_60d", "EQR_Equity_ret_1d", "EWY_Korea_zscore_60d", "PPL_PPL_ret_1d", "HD_ret_1d", "CI_Cigna_vol_20d", "LOW_Lowes_ret_20d", "CMCSA_ret_1d", "ORCL_vol_20d", "EWM_Malaysia_ret_1d"], "is_new": true}, {"model_id": "new_h3_NORMAL_GradientBoosting_N25_t6", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 3, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "BTI_BritishAmerican_ret_20d", "heston_var_ev_h5", "GILD_Gilead_ret_20d", "SLB_Schlumberger_ret_5d", "EFFR_vol_20d", "FedFunds_zscore_60d", "EWQ_France_zscore_60d", "NOC_Northrop_ret_20d", "HangSeng_HK_ret_5d", "TXN_vol_20d", "JNJ_ret_1d", "Brent_Oil_FRED_ret_5d", "PAYX_Paychex_ret_20d", "TED_Spread_zscore_60d", "EMR_Emerson_ret_20d", "PAYX_Paychex_zscore_60d", "LOW_Lowes_ret_5d", "SCHW_Schwab_ret_5d", "QQQ_vol_20d", "EWG_Germany_ret_20d", "BLK_BlackRock_zscore_60d", "MS_MorganStanley_ret_1d", "BTI_BritishAmerican_ret_5d"], "is_new": true}, {"model_id": "new_h3_NORMAL_GradientBoosting_N25_t7", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 3, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "SCHW_Schwab_ret_5d", "EXC_Exelon_ret_1d", "CLX_Clorox_vol_20d", "VVIX_ret_20d", "MSTR_Bitcoin3_ret_1d", "HangSeng_HK_vol_20d", "AMZN_ret_5d", "BLK_BlackRock_zscore_60d", "heston_var_ev_h5", "VOD_Vodafone_zscore_60d", "EFFR_ret_1d", "IWM_SmallCap_vol_20d", "NFCI_ret_5d", "SLB_Schlumberger_ret_5d", "EWG_Germany_ret_20d", "IYM_BasicMaterials_ret_20d", "WTI_Oil_FRED_zscore_60d", "Nikkei_Japan_vol_20d", "DAX_Germany_zscore_60d", "PAYX_Paychex_vol_20d", "T10Y2Y_Spread_ret_5d", "EWY_Korea_ret_20d", "US1Y_Rate_ret_5d"], "is_new": true}, {"model_id": "new_h3_NORMAL_GradientBoosting_N30_t0", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 3, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "IYM_BasicMaterials_ret_20d", "EXC_Exelon_zscore_60d", "3M_ret_5d", "DE_Deere_vol_20d", "EWM_Malaysia_ret_1d", "HangSeng_HK_ret_1d", "PPL_PPL_ret_1d", "DAX_Germany_vol_20d", "HD_ret_20d", "LLY_zscore_60d", "Core_CPI_zscore_60d", "EWA_Australia_zscore_60d", "PAYX_Paychex_zscore_60d", "GE_ret_1d", "3M_vol_20d", "DAX_Germany_zscore_60d", "TGT_Target_zscore_60d", "EMR_Emerson_ret_20d", "NVDA_vol_20d", "LOW_Lowes_ret_20d", "Nikkei_Japan_zscore_60d", "JNJ_ret_1d", "EQR_Equity_ret_1d", "US3M_Rate_zscore_60d", "spx_abs_ret_max_5d", "MSTR_Bitcoin3_ret_1d", "Retail_Sales_zscore_60d", "LOW_Lowes_ret_5d"], "is_new": true}, {"model_id": "new_h3_NORMAL_GradientBoosting_N30_t1", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 3, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWY_Korea_zscore_60d", "JNJ_ret_1d", "EXC_Exelon_zscore_60d", "AMD_ret_5d", "EWM_Malaysia_zscore_60d", "HangSeng_HK_vol_20d", "US7Y_Rate_ret_20d", "heston_ev_h3", "SLB_Schlumberger_ret_5d", "US3Y_Rate_ret_5d", "EWY_Korea_ret_20d", "XLB_Materials_zscore_60d", "PAYX_Paychex_vol_20d", "LOW_Lowes_ret_5d", "LOW_Lowes_ret_20d", "IYM_BasicMaterials_ret_20d", "EWQ_France_ret_20d", "M_Macys_vol_20d", "HangSeng_HK_ret_1d", "EOG_EOGResources_ret_5d", "EQR_Equity_ret_1d", "AXP_Amex_vol_20d", "US6M_Rate_ret_20d", "FedFunds_zscore_60d", "ORCL_zscore_60d", "ASX_Australia_ret_5d", "XLK_Tech_zscore_60d", "EXC_Exelon_ret_1d"], "is_new": true}, {"model_id": "new_h3_NORMAL_GradientBoosting_N30_t2", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 3, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "SLB_Schlumberger_ret_1d", "EWG_Germany_ret_20d", "XLK_Tech_zscore_60d", "AMD_ret_1d", "BTI_BritishAmerican_ret_5d", "US30Y_Rate_ret_20d", "PLD_Prologis_ret_5d", "MSTR_Bitcoin3_ret_5d", "XLF_Fin_vol_20d", "ENB_EnbridgeInc_ret_1d", "PAYX_Paychex_vol_20d", "BLK_BlackRock_zscore_60d", "GE_ret_1d", "EWQ_France_zscore_60d", "SLB_Schlumberger_ret_5d", "TM_Telephone_vol_20d", "IYM_BasicMaterials_ret_20d", "DIS_vol_20d", "SPY_zscore_60d", "PAYX_Paychex_zscore_60d", "BTI_BritishAmerican_ret_20d", "LMT_LockheedMartin_vol_20d", "WTI_Oil_FRED_zscore_60d", "EWM_Malaysia_zscore_60d", "XOM_ret_1d", "GD_GeneralDynamics_zscore_60d", "EWM_Malaysia_ret_1d", "AMZN_ret_5d"], "is_new": true}, {"model_id": "new_h3_NORMAL_GradientBoosting_N30_t3", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 3, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "IBEX_Spain_ret_20d", "GE_ret_1d", "EQIX_Equinix_ret_5d", "HD_ret_1d", "BLK_BlackRock_zscore_60d", "EFFR_ret_1d", "XLV_Health_zscore_60d", "EXC_Exelon_zscore_60d", "TM_Telephone_ret_1d", "3M_vol_20d", "HangSeng_HK_ret_5d", "NWL_Newell_ret_20d", "SO_SouthernCo_ret_5d", "TED_Spread_vol_20d", "MSTR_Bitcoin3_ret_5d", "US1Y_Rate_ret_5d", "AXP_Amex_ret_20d", "TM_Telephone_vol_20d", "PAYX_Paychex_zscore_60d", "M_Macys_vol_20d", "EWG_Germany_ret_20d", "EXC_Exelon_ret_1d", "EOG_EOGResources_vol_20d", "SBUX_vol_20d", "ENB_EnbridgeInc_ret_1d", "HangSeng_HK_vol_20d", "ORCL_zscore_60d", "PLD_Prologis_ret_5d"], "is_new": true}, {"model_id": "new_h3_NORMAL_GradientBoosting_N30_t4", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 3, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "T_ret_1d", "PCAR_PaccarInc_ret_5d", "EWA_Australia_zscore_60d", "Retail_Sales_zscore_60d", "TM_Telephone_ret_1d", "CI_Cigna_vol_20d", "JNJ_ret_1d", "NEE_NextEra_ret_20d", "DAX_Germany_vol_20d", "Michigan_Sentiment_ret_20d", "EFFR_vol_20d", "BTI_BritishAmerican_ret_20d", "SBUX_zscore_60d", "EWJ_Japan_vol_20d", "MSTR_Bitcoin3_ret_1d", "QQQ_vol_20d", "hmm_p_stress", "EWQ_France_ret_20d", "TED_Spread_zscore_60d", "SCHW_Schwab_ret_5d", "EWC_Canada_zscore_60d", "XOM_ret_20d", "HD_zscore_60d", "AMT_AmericanTower_ret_1d", "EWL_Switzerland_vol_20d", "Brent_Oil_FRED_ret_5d", "US3M_Rate_zscore_60d", "Core_PCE_zscore_60d"], "is_new": true}, {"model_id": "new_h3_NORMAL_GradientBoosting_N30_t5", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 3, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "NWL_Newell_ret_20d", "EOG_EOGResources_ret_5d", "IBEX_Spain_ret_20d", "MS_MorganStanley_zscore_60d", "XLY_Disc_vol_20d", "TXN_vol_20d", "T10Y2Y_Spread_ret_5d", "US5Y_Rate_ret_5d", "Nikkei_Japan_zscore_60d", "Retail_Sales_zscore_60d", "MS_MorganStanley_ret_5d", "EOG_EOGResources_vol_20d", "Brent_Oil_FRED_ret_5d", "SJM_JM_Smucker_ret_5d", "DHR_ret_1d", "EQR_Equity_ret_1d", "spx_vol_5d", "PG_ret_20d", "US3M_Rate_zscore_60d", "XLK_Tech_zscore_60d", "DAX_Germany_zscore_60d", "EXC_Exelon_ret_1d", "SCHW_Schwab_ret_5d", "Michigan_Sentiment_ret_20d", "US1Y_Rate_ret_20d", "vix_mean_abs_ret_5d", "ORCL_zscore_60d", "AVB_AvalonBay_zscore_60d"], "is_new": true}, {"model_id": "new_h3_NORMAL_GradientBoosting_N30_t6", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 3, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "PAYX_Paychex_ret_20d", "CMCSA_ret_1d", "heston_var_ev_h3", "GD_GeneralDynamics_zscore_60d", "SPY_zscore_60d", "US3Y_Rate_ret_5d", "LOW_Lowes_ret_20d", "PFE_ret_1d", "XLK_Tech_zscore_60d", "DHR_vol_20d", "IYR_US_REIT2_zscore_60d", "PCAR_PaccarInc_ret_5d", "TM_Telephone_ret_1d", "MO_AltriaMG_ret_1d", "EOG_EOGResources_vol_20d", "EWA_Australia_zscore_60d", "INTC_ret_1d", "DOW_Price_zscore_60d", "VVIX_ret_20d", "SJM_JM_Smucker_ret_5d", "BDX_Becton_Dickinson_ret_20d", "AMZN_ret_5d", "EWA_Australia_ret_1d", "HD_zscore_60d", "BTI_BritishAmerican_ret_5d", "EWM_Malaysia_ret_1d", "vix_mean_abs_ret_5d", "EWQ_France_zscore_60d"], "is_new": true}, {"model_id": "new_h3_NORMAL_GradientBoosting_N30_t7", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 3, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "HD_ret_5d", "Retail_Sales_zscore_60d", "CPB_CampbellSoup_ret_20d", "BTI_BritishAmerican_ret_20d", "LLY_zscore_60d", "INTC_ret_1d", "SPY_zscore_60d", "ES_Evergy_ret_1d", "EQR_Equity_ret_1d", "spx_momentum_3d", "US1Y_Rate_ret_5d", "CTAS_Cintas_vol_20d", "US3M_Rate_vol_20d", "IYR_US_REIT2_zscore_60d", "CI_Cigna_vol_20d", "PAYX_Paychex_ret_20d", "CMCSA_ret_1d", "XLB_Materials_zscore_60d", "3M_ret_5d", "PAYX_Paychex_zscore_60d", "AMZN_ret_5d", "ORCL_vol_20d", "PCAR_PaccarInc_ret_5d", "PPL_PPL_ret_1d", "Nikkei_Japan_vol_20d", "EWC_Canada_zscore_60d", "Core_CPI_zscore_60d", "EWH_HongKong_ret_5d"], "is_new": true}, {"model_id": "new_h3_NORMAL_RandomForest_N5_t0", "algo": "RandomForest", "regime": "NORMAL", "horizon": 3, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWS_Singapore_ret_5d", "US1Y_Rate_ret_20d", "EWC_Canada_zscore_60d"], "is_new": true}, {"model_id": "new_h3_NORMAL_RandomForest_N5_t1", "algo": "RandomForest", "regime": "NORMAL", "horizon": 3, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "LMT_LockheedMartin_ret_1d", "ITT_ITTInc_ret_5d", "AMD_ret_1d"], "is_new": true}, {"model_id": "new_h3_NORMAL_RandomForest_N5_t2", "algo": "RandomForest", "regime": "NORMAL", "horizon": 3, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EOG_EOGResources_vol_20d", "CLX_Clorox_vol_20d", "ASX_Australia_ret_5d"], "is_new": true}, {"model_id": "new_h3_NORMAL_RandomForest_N5_t3", "algo": "RandomForest", "regime": "NORMAL", "horizon": 3, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "Michigan_Sentiment_ret_20d", "PAYX_Paychex_zscore_60d", "IWM_SmallCap_vol_20d"], "is_new": true}, {"model_id": "new_h3_NORMAL_RandomForest_N5_t4", "algo": "RandomForest", "regime": "NORMAL", "horizon": 3, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "US3Y_Rate_ret_5d", "PAYX_Paychex_vol_20d", "PPL_PPL_ret_1d"], "is_new": true}, {"model_id": "new_h3_NORMAL_RandomForest_N5_t5", "algo": "RandomForest", "regime": "NORMAL", "horizon": 3, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "3M_ret_5d", "LOW_Lowes_ret_5d", "Nikkei_Japan_vol_20d"], "is_new": true}, {"model_id": "new_h3_NORMAL_RandomForest_N5_t6", "algo": "RandomForest", "regime": "NORMAL", "horizon": 3, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "ES_Evergy_ret_1d", "AVB_AvalonBay_zscore_60d", "BTI_BritishAmerican_ret_5d"], "is_new": true}, {"model_id": "new_h3_NORMAL_RandomForest_N5_t7", "algo": "RandomForest", "regime": "NORMAL", "horizon": 3, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "MO_AltriaMG_ret_1d", "MS_MorganStanley_ret_1d", "EWA_Australia_zscore_60d"], "is_new": true}, {"model_id": "new_h3_NORMAL_RandomForest_N8_t0", "algo": "RandomForest", "regime": "NORMAL", "horizon": 3, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "SJM_JM_Smucker_ret_5d", "Nikkei_Japan_zscore_60d", "EWQ_France_ret_20d", "NVDA_vol_20d", "gjr_condvar_h1", "HangSeng_HK_ret_1d"], "is_new": true}, {"model_id": "new_h3_NORMAL_RandomForest_N8_t1", "algo": "RandomForest", "regime": "NORMAL", "horizon": 3, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "AORD_AUS_zscore_60d", "spx_vol_5d", "Michigan_Sentiment_ret_20d", "EWA_Australia_zscore_60d", "DE_Deere_ret_5d", "NFCI_ret_5d"], "is_new": true}, {"model_id": "new_h3_NORMAL_RandomForest_N8_t2", "algo": "RandomForest", "regime": "NORMAL", "horizon": 3, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "MSTR_Bitcoin3_ret_1d", "AMT_AmericanTower_ret_1d", "GILD_Gilead_ret_20d", "DHR_ret_1d", "US6M_Rate_ret_20d", "HUM_Humana_ret_5d"], "is_new": true}, {"model_id": "new_h3_NORMAL_RandomForest_N8_t3", "algo": "RandomForest", "regime": "NORMAL", "horizon": 3, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "XLV_Health_zscore_60d", "EOG_EOGResources_vol_20d", "GD_GeneralDynamics_zscore_60d", "PAYX_Paychex_vol_20d", "FedFunds_zscore_60d", "EWJ_Japan_vol_20d"], "is_new": true}, {"model_id": "new_h3_NORMAL_RandomForest_N8_t4", "algo": "RandomForest", "regime": "NORMAL", "horizon": 3, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "XLB_Materials_zscore_60d", "spx_vol_5d", "SBUX_ret_5d", "TM_Telephone_vol_20d", "EOG_EOGResources_ret_5d", "BLK_BlackRock_zscore_60d"], "is_new": true}, {"model_id": "new_h3_NORMAL_RandomForest_N8_t5", "algo": "RandomForest", "regime": "NORMAL", "horizon": 3, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "Core_PCE_zscore_60d", "CPB_CampbellSoup_ret_20d", "DHR_ret_1d", "AMGN_Amgen_ret_1d", "EWL_Switzerland_vol_20d", "SLB_Schlumberger_ret_5d"], "is_new": true}, {"model_id": "new_h3_NORMAL_RandomForest_N8_t6", "algo": "RandomForest", "regime": "NORMAL", "horizon": 3, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "MO_AltriaMG_ret_1d", "3M_ret_5d", "vix_acceleration_1d", "XOM_ret_1d", "LMT_LockheedMartin_ret_1d", "HangSeng_HK_ret_1d"], "is_new": true}, {"model_id": "new_h3_NORMAL_RandomForest_N8_t7", "algo": "RandomForest", "regime": "NORMAL", "horizon": 3, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "US30Y_Rate_ret_20d", "BLK_BlackRock_zscore_60d", "XOM_ret_20d", "MS_MorganStanley_ret_1d", "SO_SouthernCo_ret_5d", "ASX_Australia_ret_5d"], "is_new": true}, {"model_id": "new_h3_NORMAL_RandomForest_N10_t0", "algo": "RandomForest", "regime": "NORMAL", "horizon": 3, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "3M_vol_20d", "IWM_SmallCap_vol_20d", "GE_ret_1d", "NVDA_vol_20d", "T10Y2Y_Spread_ret_5d", "AMGN_Amgen_ret_1d", "CCI_CrownCastle_vol_20d", "EWG_Germany_ret_20d"], "is_new": true}, {"model_id": "new_h3_NORMAL_RandomForest_N10_t1", "algo": "RandomForest", "regime": "NORMAL", "horizon": 3, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "NOC_Northrop_ret_20d", "3M_ret_5d", "LOW_Lowes_ret_20d", "SCHW_Schwab_ret_5d", "FedFunds_zscore_60d", "EWC_Canada_zscore_60d", "ES_Evergy_ret_1d", "CPB_CampbellSoup_vol_20d"], "is_new": true}, {"model_id": "new_h3_NORMAL_RandomForest_N10_t2", "algo": "RandomForest", "regime": "NORMAL", "horizon": 3, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "GD_GeneralDynamics_zscore_60d", "US5Y_Rate_ret_5d", "IYR_US_REIT2_zscore_60d", "XLY_Disc_vol_20d", "gjr_condvar_h1", "spx_abs_ret_max_5d", "heston_var_ev_h7", "XOM_ret_1d"], "is_new": true}, {"model_id": "new_h3_NORMAL_RandomForest_N10_t3", "algo": "RandomForest", "regime": "NORMAL", "horizon": 3, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "VVIX_ret_20d", "MSTR_Bitcoin3_ret_20d", "XOM_ret_1d", "spx_momentum_3d", "AXP_Amex_vol_20d", "XLB_Materials_zscore_60d", "BTI_BritishAmerican_ret_5d", "EWQ_France_ret_20d"], "is_new": true}, {"model_id": "new_h3_NORMAL_RandomForest_N10_t4", "algo": "RandomForest", "regime": "NORMAL", "horizon": 3, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "heston_var_ev_h5", "NOC_Northrop_ret_20d", "INTC_ret_5d", "US3M_Rate_zscore_60d", "DAX_Germany_zscore_60d", "AMGN_Amgen_ret_1d", "MO_AltriaMG_ret_1d", "US3Y_Rate_ret_5d"], "is_new": true}, {"model_id": "new_h3_NORMAL_RandomForest_N10_t5", "algo": "RandomForest", "regime": "NORMAL", "horizon": 3, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "US3M_Rate_zscore_60d", "AMZN_ret_5d", "Brent_Oil_FRED_ret_5d", "NVDA_vol_20d", "T10Y2Y_Spread_ret_5d", "AMD_ret_5d", "CPB_CampbellSoup_zscore_60d", "CI_Cigna_vol_20d"], "is_new": true}, {"model_id": "new_h3_NORMAL_RandomForest_N10_t6", "algo": "RandomForest", "regime": "NORMAL", "horizon": 3, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "XOM_ret_1d", "MSTR_Bitcoin3_ret_5d", "HD_zscore_60d", "EWH_HongKong_ret_5d", "XLF_Fin_vol_20d", "HangSeng_HK_ret_1d", "LUV_SouthwestAir_ret_5d", "gjr_condvar_h1"], "is_new": true}, {"model_id": "new_h3_NORMAL_RandomForest_N10_t7", "algo": "RandomForest", "regime": "NORMAL", "horizon": 3, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "DAX_Germany_zscore_60d", "BLK_BlackRock_zscore_60d", "3M_ret_5d", "US7Y_Rate_ret_20d", "NFCI_ret_5d", "ASX_Australia_ret_5d", "Core_CPI_zscore_60d", "MRK_Merck_zscore_60d"], "is_new": true}, {"model_id": "new_h3_NORMAL_RandomForest_N12_t0", "algo": "RandomForest", "regime": "NORMAL", "horizon": 3, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "LOW_Lowes_ret_20d", "EWL_Switzerland_zscore_60d", "EQIX_Equinix_ret_5d", "LLY_zscore_60d", "Nikkei_Japan_zscore_60d", "EOG_EOGResources_vol_20d", "HangSeng_HK_vol_20d", "US30Y_Rate_ret_20d", "TXN_vol_20d", "heston_var_ev_h7"], "is_new": true}, {"model_id": "new_h3_NORMAL_RandomForest_N12_t1", "algo": "RandomForest", "regime": "NORMAL", "horizon": 3, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "AXP_Amex_ret_20d", "FedFunds_zscore_60d", "NFCI_ret_5d", "GILD_Gilead_ret_20d", "US1Y_Rate_ret_20d", "IYM_BasicMaterials_ret_20d", "ITT_ITTInc_ret_5d", "LOW_Lowes_ret_20d", "CPB_CampbellSoup_ret_20d", "Michigan_Sentiment_ret_20d"], "is_new": true}, {"model_id": "new_h3_NORMAL_RandomForest_N12_t2", "algo": "RandomForest", "regime": "NORMAL", "horizon": 3, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "ASX_Australia_ret_5d", "IYR_US_REIT2_zscore_60d", "heston_var_ev_h3", "PAYX_Paychex_zscore_60d", "HD_zscore_60d", "CPB_CampbellSoup_ret_5d", "CCI_CrownCastle_vol_20d", "US3M_Rate_vol_20d", "EWG_Germany_vol_20d", "MO_AltriaMG_ret_1d"], "is_new": true}, {"model_id": "new_h3_NORMAL_RandomForest_N12_t3", "algo": "RandomForest", "regime": "NORMAL", "horizon": 3, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "SO_SouthernCo_ret_5d", "heston_var_ev_h5", "US6M_Rate_ret_20d", "Brent_Oil_FRED_ret_20d", "MS_MorganStanley_zscore_60d", "M_Macys_vol_20d", "SJM_JM_Smucker_ret_1d", "ORCL_zscore_60d", "AMZN_ret_5d", "XOM_ret_1d"], "is_new": true}, {"model_id": "new_h3_NORMAL_RandomForest_N12_t4", "algo": "RandomForest", "regime": "NORMAL", "horizon": 3, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "SLB_Schlumberger_ret_1d", "IYR_US_REIT2_zscore_60d", "MSTR_Bitcoin3_ret_1d", "TED_Spread_zscore_60d", "AVB_AvalonBay_zscore_60d", "EXC_Exelon_zscore_60d", "EWA_Australia_ret_1d", "XLF_Fin_vol_20d", "DHR_vol_20d", "VRP_ma5"], "is_new": true}, {"model_id": "new_h3_NORMAL_RandomForest_N12_t5", "algo": "RandomForest", "regime": "NORMAL", "horizon": 3, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "TM_Telephone_vol_20d", "heston_ev_h3", "IWM_SmallCap_vol_20d", "SBUX_ret_5d", "SO_SouthernCo_ret_5d", "EWG_Germany_ret_20d", "XLF_Fin_vol_20d", "TXN_vol_20d", "AXP_Amex_ret_20d", "EWA_Australia_zscore_60d"], "is_new": true}, {"model_id": "new_h3_NORMAL_RandomForest_N12_t6", "algo": "RandomForest", "regime": "NORMAL", "horizon": 3, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "PPL_PPL_ret_1d", "EWG_Germany_vol_20d", "CI_Cigna_vol_20d", "EFFR_ret_1d", "BA_ret_1d", "MO_AltriaMG_ret_1d", "US1Y_Rate_ret_20d", "US1Y_Rate_ret_5d", "NEE_NextEra_ret_20d", "US3M_Rate_zscore_60d"], "is_new": true}, {"model_id": "new_h3_NORMAL_RandomForest_N12_t7", "algo": "RandomForest", "regime": "NORMAL", "horizon": 3, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "BLK_BlackRock_zscore_60d", "PCAR_PaccarInc_ret_5d", "EWY_Korea_zscore_60d", "T10Y2Y_Spread_ret_5d", "PFE_ret_1d", "US1Y_Rate_ret_20d", "Brent_Oil_FRED_ret_5d", "EWA_Australia_ret_1d", "EXC_Exelon_ret_1d", "AMT_AmericanTower_ret_1d"], "is_new": true}, {"model_id": "new_h3_NORMAL_RandomForest_N15_t0", "algo": "RandomForest", "regime": "NORMAL", "horizon": 3, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "AMD_ret_5d", "INTC_ret_5d", "BDX_Becton_Dickinson_ret_20d", "Brent_Oil_FRED_ret_20d", "US1Y_Rate_ret_20d", "CPB_CampbellSoup_ret_5d", "EWS_Singapore_ret_5d", "MSTR_Bitcoin3_ret_5d", "Brent_Oil_FRED_ret_5d", "SCHW_Schwab_ret_5d", "ASX_Australia_ret_5d", "3M_ret_5d", "ASX_Australia_vol_20d"], "is_new": true}, {"model_id": "new_h3_NORMAL_RandomForest_N15_t1", "algo": "RandomForest", "regime": "NORMAL", "horizon": 3, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "DE_Deere_vol_20d", "EWA_Australia_zscore_60d", "vix_acceleration_1d", "EWY_Korea_zscore_60d", "SLB_Schlumberger_ret_1d", "CPB_CampbellSoup_vol_20d", "AMD_ret_1d", "US7Y_Rate_ret_20d", "HD_ret_20d", "IWM_SmallCap_vol_20d", "ENB_EnbridgeInc_ret_1d", "INTC_ret_5d", "EWY_Korea_ret_20d"], "is_new": true}, {"model_id": "new_h3_NORMAL_RandomForest_N15_t2", "algo": "RandomForest", "regime": "NORMAL", "horizon": 3, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "LOW_Lowes_ret_5d", "Nikkei_Japan_zscore_60d", "CCI_CrownCastle_vol_20d", "GE_ret_1d", "GD_GeneralDynamics_zscore_60d", "CPB_CampbellSoup_ret_5d", "EFFR_ret_1d", "spx_vol_5d", "US3M_Rate_vol_20d", "TM_Telephone_vol_20d", "PAYX_Paychex_vol_20d", "BLK_BlackRock_zscore_60d", "BTI_BritishAmerican_ret_20d"], "is_new": true}, {"model_id": "new_h3_NORMAL_RandomForest_N15_t3", "algo": "RandomForest", "regime": "NORMAL", "horizon": 3, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "IYR_US_REIT2_zscore_60d", "DIS_vol_20d", "VOD_Vodafone_zscore_60d", "EXC_Exelon_ret_1d", "CPB_CampbellSoup_zscore_60d", "SJM_JM_Smucker_ret_1d", "Nikkei_Japan_zscore_60d", "CTAS_Cintas_vol_20d", "CPB_CampbellSoup_ret_20d", "HangSeng_HK_vol_20d", "SO_SouthernCo_ret_5d", "EWM_Malaysia_zscore_60d", "EFFR_vol_20d"], "is_new": true}, {"model_id": "new_h3_NORMAL_RandomForest_N15_t4", "algo": "RandomForest", "regime": "NORMAL", "horizon": 3, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "DE_Deere_vol_20d", "NFCI_ret_5d", "Brent_Oil_FRED_ret_5d", "DE_Deere_ret_5d", "DAX_Germany_vol_20d", "MS_MorganStanley_ret_1d", "US3M_Rate_vol_20d", "HangSeng_HK_ret_1d", "SPY_zscore_60d", "CLX_Clorox_vol_20d", "heston_var_ev_h3", "BLK_BlackRock_zscore_60d", "3M_ret_5d"], "is_new": true}, {"model_id": "new_h3_NORMAL_RandomForest_N15_t5", "algo": "RandomForest", "regime": "NORMAL", "horizon": 3, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWL_Switzerland_zscore_60d", "AMT_AmericanTower_ret_1d", "DAX_Germany_vol_20d", "NVDA_vol_20d", "PCAR_PaccarInc_ret_5d", "US6M_Rate_ret_20d", "MSTR_Bitcoin3_ret_20d", "Industrial_Production_zscore_60d", "MO_AltriaMG_ret_1d", "XOM_ret_20d", "MS_MorganStanley_ret_1d", "hmm_p_stress", "PG_ret_20d"], "is_new": true}, {"model_id": "new_h3_NORMAL_RandomForest_N15_t6", "algo": "RandomForest", "regime": "NORMAL", "horizon": 3, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "MS_MorganStanley_ret_1d", "SLB_Schlumberger_ret_1d", "HangSeng_HK_vol_20d", "CPB_CampbellSoup_ret_5d", "HUM_Humana_ret_5d", "EWQ_France_zscore_60d", "HD_ret_20d", "MS_MorganStanley_ret_5d", "EOG_EOGResources_ret_5d", "EOG_EOGResources_vol_20d", "AMT_AmericanTower_ret_1d", "SBUX_vol_20d", "FedFunds_zscore_60d"], "is_new": true}, {"model_id": "new_h3_NORMAL_RandomForest_N15_t7", "algo": "RandomForest", "regime": "NORMAL", "horizon": 3, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "SO_SouthernCo_ret_5d", "US3Y_Rate_ret_5d", "heston_var_ev_h5", "AMD_ret_5d", "JNJ_ret_1d", "XLV_Health_zscore_60d", "MO_AltriaMG_ret_1d", "3M_vol_20d", "GE_ret_1d", "EMR_Emerson_ret_20d", "TED_Spread_vol_20d", "BTI_BritishAmerican_ret_5d", "ORCL_vol_20d"], "is_new": true}, {"model_id": "new_h3_NORMAL_RandomForest_N20_t0", "algo": "RandomForest", "regime": "NORMAL", "horizon": 3, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "IYM_BasicMaterials_ret_20d", "PAYX_Paychex_vol_20d", "SBUX_zscore_60d", "EQIX_Equinix_ret_5d", "CI_Cigna_vol_20d", "AMZN_ret_5d", "SJM_JM_Smucker_ret_5d", "ES_Evergy_ret_1d", "PLD_Prologis_ret_5d", "HD_ret_5d", "DAX_Germany_vol_20d", "AXP_Amex_vol_20d", "MRK_Merck_zscore_60d", "US7Y_Rate_ret_20d", "CCI_CrownCastle_vol_20d", "LMT_LockheedMartin_ret_1d", "PFE_ret_1d", "US3Y_Rate_ret_5d"], "is_new": true}, {"model_id": "new_h3_NORMAL_RandomForest_N20_t1", "algo": "RandomForest", "regime": "NORMAL", "horizon": 3, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "Industrial_Production_zscore_60d", "CMCSA_ret_1d", "US1Y_Rate_ret_20d", "IBEX_Spain_ret_20d", "VRP_ma5", "LOW_Lowes_ret_20d", "MO_AltriaMG_ret_1d", "ENB_EnbridgeInc_ret_1d", "XOM_ret_1d", "NWL_Newell_ret_20d", "EXC_Exelon_ret_1d", "EWY_Korea_ret_20d", "XOM_ret_20d", "INTC_ret_5d", "DHR_ret_1d", "CPB_CampbellSoup_ret_20d", "SBUX_ret_5d", "EWC_Canada_zscore_60d"], "is_new": true}, {"model_id": "new_h3_NORMAL_RandomForest_N20_t2", "algo": "RandomForest", "regime": "NORMAL", "horizon": 3, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "TED_Spread_vol_20d", "Retail_Sales_zscore_60d", "heston_var_ev_h5", "vix_mean_abs_ret_5d", "EOG_EOGResources_ret_5d", "US3M_Rate_zscore_60d", "TGT_Target_zscore_60d", "BLK_BlackRock_zscore_60d", "spx_vol_5d", "EMR_Emerson_ret_20d", "AMT_AmericanTower_ret_1d", "EFFR_ret_1d", "XLY_Disc_vol_20d", "Brent_Oil_FRED_ret_5d", "PFE_ret_1d", "EOG_EOGResources_vol_20d", "MSTR_Bitcoin3_ret_5d", "INTC_ret_1d"], "is_new": true}, {"model_id": "new_h3_NORMAL_RandomForest_N20_t3", "algo": "RandomForest", "regime": "NORMAL", "horizon": 3, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "ASX_Australia_ret_5d", "IBEX_Spain_ret_20d", "XOM_ret_20d", "IYR_US_REIT2_zscore_60d", "SBUX_zscore_60d", "SPY_zscore_60d", "CTAS_Cintas_vol_20d", "EWS_Singapore_ret_5d", "CPB_CampbellSoup_zscore_60d", "HD_zscore_60d", "LOW_Lowes_ret_5d", "MSTR_Bitcoin3_ret_20d", "DHR_ret_1d", "MRK_Merck_zscore_60d", "EWM_Malaysia_vol_20d", "BLK_BlackRock_zscore_60d", "EWL_Switzerland_vol_20d", "HD_ret_1d"], "is_new": true}, {"model_id": "new_h3_NORMAL_RandomForest_N20_t4", "algo": "RandomForest", "regime": "NORMAL", "horizon": 3, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "Core_CPI_zscore_60d", "CTAS_Cintas_vol_20d", "EWH_HongKong_ret_5d", "BTI_BritishAmerican_ret_20d", "HD_ret_20d", "NWL_Newell_ret_20d", "HangSeng_HK_ret_5d", "HangSeng_HK_vol_20d", "ES_Evergy_ret_1d", "CI_Cigna_vol_20d", "EWY_Korea_ret_20d", "PLD_Prologis_ret_5d", "heston_var_ev_h5", "Retail_Sales_zscore_60d", "SJM_JM_Smucker_ret_5d", "AORD_AUS_zscore_60d", "MRK_Merck_zscore_60d", "IYR_US_REIT2_zscore_60d"], "is_new": true}, {"model_id": "new_h3_NORMAL_RandomForest_N20_t5", "algo": "RandomForest", "regime": "NORMAL", "horizon": 3, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "QQQ_vol_20d", "AMZN_ret_5d", "spx_vol_5d", "DAX_Germany_zscore_60d", "Nikkei_Japan_vol_20d", "BA_ret_1d", "HangSeng_HK_vol_20d", "US3M_Rate_vol_20d", "PLD_Prologis_ret_5d", "M_Macys_vol_20d", "XOM_ret_1d", "hmm_p_stress", "XLK_Tech_zscore_60d", "US7Y_Rate_ret_20d", "SPY_zscore_60d", "PAYX_Paychex_zscore_60d", "ES_Evergy_ret_1d", "LMT_LockheedMartin_ret_1d"], "is_new": true}, {"model_id": "new_h3_NORMAL_RandomForest_N20_t6", "algo": "RandomForest", "regime": "NORMAL", "horizon": 3, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "BTI_BritishAmerican_ret_20d", "EWJ_Japan_vol_20d", "LMT_LockheedMartin_ret_1d", "AMD_ret_5d", "PAYX_Paychex_vol_20d", "ORCL_vol_20d", "HD_zscore_60d", "XLY_Disc_vol_20d", "XOM_ret_1d", "MSTR_Bitcoin3_ret_20d", "PAYX_Paychex_zscore_60d", "ENB_EnbridgeInc_ret_1d", "Michigan_Sentiment_ret_20d", "EWA_Australia_ret_1d", "EWL_Switzerland_vol_20d", "US3M_Rate_vol_20d", "NFCI_ret_5d", "SLB_Schlumberger_ret_1d"], "is_new": true}, {"model_id": "new_h3_NORMAL_RandomForest_N20_t7", "algo": "RandomForest", "regime": "NORMAL", "horizon": 3, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "CPB_CampbellSoup_zscore_60d", "XLV_Health_zscore_60d", "HD_ret_20d", "SJM_JM_Smucker_ret_1d", "INTC_ret_1d", "AXP_Amex_vol_20d", "spx_momentum_3d", "LOW_Lowes_ret_20d", "heston_ev_h3", "ES_Evergy_ret_1d", "DAX_Germany_zscore_60d", "AXP_Amex_ret_20d", "CPB_CampbellSoup_ret_20d", "BTI_BritishAmerican_ret_20d", "DHR_vol_20d", "SBUX_ret_5d", "AMD_ret_1d", "MO_AltriaMG_ret_1d"], "is_new": true}, {"model_id": "new_h3_NORMAL_RandomForest_N25_t0", "algo": "RandomForest", "regime": "NORMAL", "horizon": 3, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWH_HongKong_ret_5d", "US1Y_Rate_ret_20d", "CI_Cigna_vol_20d", "HangSeng_HK_ret_5d", "EWM_Malaysia_zscore_60d", "EWJ_Japan_vol_20d", "CTAS_Cintas_vol_20d", "NEE_NextEra_ret_20d", "Brent_Oil_FRED_ret_5d", "Nikkei_Japan_zscore_60d", "Michigan_Sentiment_ret_20d", "XOM_ret_1d", "PPL_PPL_ret_1d", "HangSeng_HK_vol_20d", "XLY_Disc_vol_20d", "NFCI_ret_5d", "US7Y_Rate_ret_20d", "SBUX_vol_20d", "AVB_AvalonBay_zscore_60d", "XLB_Materials_zscore_60d", "Core_CPI_zscore_60d", "heston_var_ev_h7", "TED_Spread_vol_20d"], "is_new": true}, {"model_id": "new_h3_NORMAL_RandomForest_N25_t1", "algo": "RandomForest", "regime": "NORMAL", "horizon": 3, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "Core_PCE_zscore_60d", "MO_AltriaMG_ret_1d", "SCHW_Schwab_ret_5d", "TXN_vol_20d", "PCAR_PaccarInc_ret_5d", "DAX_Germany_zscore_60d", "EWQ_France_zscore_60d", "US7Y_Rate_ret_20d", "BLK_BlackRock_zscore_60d", "EWC_Canada_zscore_60d", "spx_vol_5d", "vix_mean_abs_ret_5d", "ITT_ITTInc_ret_5d", "vix_acceleration_1d", "SPY_zscore_60d", "M_Macys_vol_20d", "AMD_ret_1d", "EQR_Equity_ret_1d", "XLK_Tech_zscore_60d", "HangSeng_HK_ret_5d", "DE_Deere_ret_5d", "EWH_HongKong_ret_5d", "Michigan_Sentiment_ret_20d"], "is_new": true}, {"model_id": "new_h3_NORMAL_RandomForest_N25_t2", "algo": "RandomForest", "regime": "NORMAL", "horizon": 3, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "MSTR_Bitcoin3_ret_1d", "US6M_Rate_ret_20d", "EWG_Germany_vol_20d", "PAYX_Paychex_vol_20d", "BTI_BritishAmerican_ret_5d", "HD_ret_1d", "PG_ret_20d", "US1Y_Rate_ret_20d", "SJM_JM_Smucker_ret_1d", "LLY_zscore_60d", "AMZN_ret_5d", "SPY_zscore_60d", "AMT_AmericanTower_ret_1d", "LOW_Lowes_ret_20d", "PFE_ret_1d", "BTI_BritishAmerican_ret_20d", "spx_abs_ret_max_5d", "heston_var_ev_h3", "TED_Spread_zscore_60d", "EXC_Exelon_zscore_60d", "HangSeng_HK_vol_20d", "hmm_p_stress", "AXP_Amex_vol_20d"], "is_new": true}, {"model_id": "new_h3_NORMAL_RandomForest_N25_t3", "algo": "RandomForest", "regime": "NORMAL", "horizon": 3, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "SLB_Schlumberger_ret_5d", "XLB_Materials_zscore_60d", "EXC_Exelon_ret_1d", "EOG_EOGResources_ret_5d", "GILD_Gilead_ret_20d", "HangSeng_HK_vol_20d", "EWM_Malaysia_zscore_60d", "PAYX_Paychex_ret_20d", "XLF_Fin_vol_20d", "EWG_Germany_vol_20d", "Michigan_Sentiment_ret_20d", "MS_MorganStanley_ret_1d", "Nikkei_Japan_vol_20d", "BA_ret_1d", "HD_ret_5d", "IBEX_Spain_ret_20d", "TM_Telephone_ret_1d", "MS_MorganStanley_zscore_60d", "PFE_ret_1d", "Core_CPI_zscore_60d", "Core_PCE_zscore_60d", "heston_var_ev_h3", "TED_Spread_vol_20d"], "is_new": true}, {"model_id": "new_h3_NORMAL_RandomForest_N25_t4", "algo": "RandomForest", "regime": "NORMAL", "horizon": 3, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "SBUX_zscore_60d", "ORCL_zscore_60d", "heston_var_ev_h7", "EWL_Switzerland_zscore_60d", "HUM_Humana_ret_5d", "PAYX_Paychex_vol_20d", "PG_ret_20d", "HD_ret_5d", "VOD_Vodafone_zscore_60d", "NOC_Northrop_ret_20d", "vix_acceleration_1d", "SPY_zscore_60d", "HD_zscore_60d", "MRK_Merck_zscore_60d", "DAX_Germany_zscore_60d", "spx_vol_5d", "EWQ_France_ret_20d", "EWS_Singapore_ret_5d", "CPB_CampbellSoup_vol_20d", "XOM_ret_20d", "T10Y2Y_Spread_ret_5d", "WTI_Oil_FRED_zscore_60d", "INTC_ret_5d"], "is_new": true}, {"model_id": "new_h3_NORMAL_RandomForest_N25_t5", "algo": "RandomForest", "regime": "NORMAL", "horizon": 3, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "Core_PCE_zscore_60d", "EWJ_Japan_vol_20d", "IYR_US_REIT2_zscore_60d", "GILD_Gilead_ret_20d", "XOM_ret_20d", "heston_ev_h3", "LMT_LockheedMartin_vol_20d", "JNJ_ret_1d", "ORCL_zscore_60d", "VVIX_ret_20d", "PCAR_PaccarInc_ret_5d", "AORD_AUS_zscore_60d", "PFE_ret_1d", "WTI_Oil_FRED_zscore_60d", "BLK_BlackRock_zscore_60d", "QQQ_vol_20d", "PAYX_Paychex_ret_20d", "BDX_Becton_Dickinson_ret_20d", "IBEX_Spain_ret_20d", "MRK_Merck_zscore_60d", "Industrial_Production_zscore_60d", "EWH_HongKong_ret_5d", "AMZN_ret_5d"], "is_new": true}, {"model_id": "new_h3_NORMAL_RandomForest_N25_t6", "algo": "RandomForest", "regime": "NORMAL", "horizon": 3, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "CPB_CampbellSoup_vol_20d", "FedFunds_zscore_60d", "vix_mean_abs_ret_5d", "DOW_Price_zscore_60d", "PPL_PPL_ret_1d", "EOG_EOGResources_vol_20d", "LMT_LockheedMartin_ret_1d", "VOD_Vodafone_zscore_60d", "hmm_p_stress", "CPB_CampbellSoup_zscore_60d", "EQR_Equity_ret_1d", "US5Y_Rate_ret_5d", "US1Y_Rate_ret_20d", "LOW_Lowes_ret_20d", "MSTR_Bitcoin3_ret_20d", "PAYX_Paychex_vol_20d", "DAX_Germany_zscore_60d", "EXC_Exelon_zscore_60d", "BA_ret_1d", "LUV_SouthwestAir_ret_5d", "VVIX_ret_20d", "JNJ_ret_1d", "EMR_Emerson_ret_20d"], "is_new": true}, {"model_id": "new_h3_NORMAL_RandomForest_N25_t7", "algo": "RandomForest", "regime": "NORMAL", "horizon": 3, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "HD_ret_5d", "heston_ev_h3", "TGT_Target_zscore_60d", "LMT_LockheedMartin_vol_20d", "Nikkei_Japan_zscore_60d", "ITT_ITTInc_ret_5d", "EFFR_ret_1d", "Michigan_Sentiment_ret_20d", "HangSeng_HK_vol_20d", "heston_var_ev_h7", "GE_ret_1d", "EWL_Switzerland_vol_20d", "ORCL_zscore_60d", "heston_var_ev_h3", "vix_acceleration_1d", "SLB_Schlumberger_ret_1d", "DE_Deere_vol_20d", "XLV_Health_zscore_60d", "AMZN_ret_5d", "ES_Evergy_ret_1d", "DAX_Germany_zscore_60d", "BA_ret_1d", "GILD_Gilead_ret_20d"], "is_new": true}, {"model_id": "new_h3_NORMAL_RandomForest_N30_t0", "algo": "RandomForest", "regime": "NORMAL", "horizon": 3, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "US3Y_Rate_ret_5d", "Michigan_Sentiment_ret_20d", "AVB_AvalonBay_zscore_60d", "PFE_ret_1d", "EMR_Emerson_ret_20d", "MSTR_Bitcoin3_ret_1d", "JNJ_ret_1d", "EFFR_vol_20d", "BLK_BlackRock_zscore_60d", "CPB_CampbellSoup_zscore_60d", "HD_ret_1d", "SO_SouthernCo_ret_5d", "PG_ret_20d", "NVDA_vol_20d", "AXP_Amex_vol_20d", "NFCI_ret_5d", "PAYX_Paychex_zscore_60d", "EWM_Malaysia_zscore_60d", "ORCL_zscore_60d", "EOG_EOGResources_ret_5d", "MRK_Merck_zscore_60d", "ASX_Australia_ret_5d", "BDX_Becton_Dickinson_ret_20d", "vix_mean_abs_ret_5d", "EWQ_France_zscore_60d", "spx_abs_ret_max_5d", "Core_PCE_zscore_60d", "heston_var_ev_h7"], "is_new": true}, {"model_id": "new_h3_NORMAL_RandomForest_N30_t1", "algo": "RandomForest", "regime": "NORMAL", "horizon": 3, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "MRK_Merck_zscore_60d", "3M_ret_5d", "NWL_Newell_ret_20d", "LLY_zscore_60d", "AVB_AvalonBay_zscore_60d", "MS_MorganStanley_zscore_60d", "US3M_Rate_zscore_60d", "MS_MorganStanley_ret_1d", "XOM_ret_20d", "Core_CPI_zscore_60d", "IYR_US_REIT2_zscore_60d", "TM_Telephone_vol_20d", "T10Y2Y_Spread_ret_5d", "EWY_Korea_zscore_60d", "GE_ret_1d", "LMT_LockheedMartin_ret_1d", "DAX_Germany_zscore_60d", "EWL_Switzerland_vol_20d", "hmm_p_stress", "XLY_Disc_vol_20d", "US6M_Rate_ret_20d", "FedFunds_zscore_60d", "Brent_Oil_FRED_ret_20d", "CPB_CampbellSoup_vol_20d", "AMZN_ret_5d", "ASX_Australia_ret_5d", "EOG_EOGResources_ret_5d", "US3Y_Rate_ret_5d"], "is_new": true}, {"model_id": "new_h3_NORMAL_RandomForest_N30_t2", "algo": "RandomForest", "regime": "NORMAL", "horizon": 3, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "ASX_Australia_ret_5d", "HangSeng_HK_ret_1d", "BDX_Becton_Dickinson_ret_20d", "NFCI_ret_5d", "DHR_vol_20d", "EWS_Singapore_ret_5d", "ES_Evergy_ret_1d", "CPB_CampbellSoup_ret_20d", "LOW_Lowes_ret_5d", "Core_PCE_zscore_60d", "PLD_Prologis_ret_5d", "INTC_ret_1d", "DAX_Germany_vol_20d", "XLB_Materials_zscore_60d", "MSTR_Bitcoin3_ret_1d", "QQQ_vol_20d", "VRP_ma5", "EQIX_Equinix_ret_5d", "US5Y_Rate_ret_5d", "NEE_NextEra_ret_20d", "AORD_AUS_zscore_60d", "Industrial_Production_zscore_60d", "T10Y2Y_Spread_ret_5d", "Nikkei_Japan_zscore_60d", "LMT_LockheedMartin_vol_20d", "EWM_Malaysia_zscore_60d", "CCI_CrownCastle_vol_20d", "GE_ret_1d"], "is_new": true}, {"model_id": "new_h3_NORMAL_RandomForest_N30_t3", "algo": "RandomForest", "regime": "NORMAL", "horizon": 3, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EQIX_Equinix_ret_5d", "ASX_Australia_ret_5d", "SCHW_Schwab_ret_5d", "AXP_Amex_ret_20d", "HD_ret_5d", "EWL_Switzerland_vol_20d", "CCI_CrownCastle_vol_20d", "spx_momentum_3d", "PCAR_PaccarInc_ret_5d", "PFE_ret_1d", "EOG_EOGResources_ret_5d", "ENB_EnbridgeInc_ret_1d", "DHR_vol_20d", "US30Y_Rate_ret_20d", "PLD_Prologis_ret_5d", "EWY_Korea_zscore_60d", "VVIX_ret_20d", "IYR_US_REIT2_zscore_60d", "SLB_Schlumberger_ret_1d", "US3M_Rate_zscore_60d", "EWA_Australia_zscore_60d", "M_Macys_vol_20d", "SBUX_vol_20d", "HUM_Humana_ret_5d", "XOM_ret_20d", "TM_Telephone_vol_20d", "MS_MorganStanley_zscore_60d", "vix_acceleration_1d"], "is_new": true}, {"model_id": "new_h3_NORMAL_RandomForest_N30_t4", "algo": "RandomForest", "regime": "NORMAL", "horizon": 3, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "vix_acceleration_1d", "TED_Spread_zscore_60d", "PAYX_Paychex_zscore_60d", "EWJ_Japan_vol_20d", "SJM_JM_Smucker_ret_5d", "SPY_zscore_60d", "BDX_Becton_Dickinson_ret_20d", "Core_PCE_zscore_60d", "CPB_CampbellSoup_vol_20d", "BTI_BritishAmerican_ret_20d", "EWG_Germany_vol_20d", "ASX_Australia_ret_5d", "LMT_LockheedMartin_ret_1d", "EOG_EOGResources_vol_20d", "US6M_Rate_ret_20d", "DOW_Price_zscore_60d", "EWH_HongKong_ret_5d", "vix_mean_abs_ret_5d", "EWL_Switzerland_zscore_60d", "LUV_SouthwestAir_ret_5d", "ENB_EnbridgeInc_ret_1d", "DIS_vol_20d", "XLF_Fin_vol_20d", "PLD_Prologis_ret_5d", "SCHW_Schwab_ret_5d", "IYR_US_REIT2_zscore_60d", "AXP_Amex_ret_20d", "GE_ret_1d"], "is_new": true}, {"model_id": "new_h3_NORMAL_RandomForest_N30_t5", "algo": "RandomForest", "regime": "NORMAL", "horizon": 3, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "gjr_condvar_h1", "ES_Evergy_ret_1d", "FedFunds_zscore_60d", "HD_ret_1d", "T_ret_1d", "XLK_Tech_zscore_60d", "M_Macys_vol_20d", "spx_abs_ret_max_5d", "EWY_Korea_zscore_60d", "NEE_NextEra_ret_20d", "ASX_Australia_ret_5d", "spx_momentum_3d", "CTAS_Cintas_vol_20d", "EOG_EOGResources_ret_5d", "NVDA_vol_20d", "CPB_CampbellSoup_vol_20d", "MS_MorganStanley_zscore_60d", "LUV_SouthwestAir_ret_5d", "IWM_SmallCap_vol_20d", "IBEX_Spain_ret_20d", "DE_Deere_ret_5d", "PLD_Prologis_ret_5d", "US7Y_Rate_ret_20d", "EWY_Korea_ret_20d", "EWC_Canada_zscore_60d", "US3Y_Rate_ret_5d", "PPL_PPL_ret_1d", "BDX_Becton_Dickinson_ret_20d"], "is_new": true}, {"model_id": "new_h3_NORMAL_RandomForest_N30_t6", "algo": "RandomForest", "regime": "NORMAL", "horizon": 3, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "Brent_Oil_FRED_ret_5d", "MSTR_Bitcoin3_ret_5d", "XLK_Tech_zscore_60d", "ASX_Australia_ret_5d", "IBEX_Spain_ret_20d", "QQQ_vol_20d", "MSTR_Bitcoin3_ret_20d", "Brent_Oil_FRED_ret_20d", "AXP_Amex_vol_20d", "DOW_Price_zscore_60d", "spx_vol_5d", "Nikkei_Japan_vol_20d", "LUV_SouthwestAir_ret_5d", "FedFunds_zscore_60d", "HUM_Humana_ret_5d", "CI_Cigna_vol_20d", "VOD_Vodafone_zscore_60d", "AXP_Amex_ret_20d", "XLY_Disc_vol_20d", "EWC_Canada_zscore_60d", "SBUX_zscore_60d", "PLD_Prologis_ret_5d", "spx_momentum_3d", "ENB_EnbridgeInc_ret_1d", "NOC_Northrop_ret_20d", "PFE_ret_1d", "US7Y_Rate_ret_20d", "IYM_BasicMaterials_ret_20d"], "is_new": true}, {"model_id": "new_h3_NORMAL_RandomForest_N30_t7", "algo": "RandomForest", "regime": "NORMAL", "horizon": 3, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EOG_EOGResources_ret_5d", "EWQ_France_ret_20d", "PAYX_Paychex_zscore_60d", "ES_Evergy_ret_1d", "GILD_Gilead_ret_20d", "MS_MorganStanley_zscore_60d", "GD_GeneralDynamics_zscore_60d", "IWM_SmallCap_vol_20d", "hmm_p_stress", "AMD_ret_1d", "HUM_Humana_ret_5d", "TM_Telephone_ret_1d", "LOW_Lowes_ret_20d", "CLX_Clorox_vol_20d", "CI_Cigna_vol_20d", "US7Y_Rate_ret_20d", "VRP_ma5", "ASX_Australia_ret_5d", "Michigan_Sentiment_ret_20d", "INTC_ret_5d", "MS_MorganStanley_ret_5d", "vix_acceleration_1d", "EQIX_Equinix_ret_5d", "spx_momentum_3d", "WTI_Oil_FRED_zscore_60d", "SBUX_vol_20d", "Brent_Oil_FRED_ret_20d", "Core_CPI_zscore_60d"], "is_new": true}, {"model_id": "new_h3_NORMAL_LogisticRegression_N5_t0", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 3, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EQR_Equity_ret_1d", "LLY_zscore_60d", "XLB_Materials_zscore_60d"], "is_new": true}, {"model_id": "new_h3_NORMAL_LogisticRegression_N5_t1", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 3, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "US6M_Rate_ret_20d", "Brent_Oil_FRED_ret_5d", "Michigan_Sentiment_ret_20d"], "is_new": true}, {"model_id": "new_h3_NORMAL_LogisticRegression_N5_t2", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 3, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "DE_Deere_ret_5d", "XLF_Fin_vol_20d", "DHR_vol_20d"], "is_new": true}, {"model_id": "new_h3_NORMAL_LogisticRegression_N5_t3", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 3, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "US1Y_Rate_ret_20d", "PAYX_Paychex_ret_20d", "IYM_BasicMaterials_ret_20d"], "is_new": true}, {"model_id": "new_h3_NORMAL_LogisticRegression_N5_t4", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 3, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "Michigan_Sentiment_ret_20d", "MRK_Merck_zscore_60d", "SBUX_vol_20d"], "is_new": true}, {"model_id": "new_h3_NORMAL_LogisticRegression_N5_t5", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 3, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "T10Y2Y_Spread_ret_5d", "EWQ_France_ret_20d", "ORCL_vol_20d"], "is_new": true}, {"model_id": "new_h3_NORMAL_LogisticRegression_N5_t6", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 3, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWQ_France_ret_20d", "spx_vol_5d", "HD_ret_20d"], "is_new": true}, {"model_id": "new_h3_NORMAL_LogisticRegression_N5_t7", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 3, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "SLB_Schlumberger_ret_5d", "EWL_Switzerland_vol_20d", "MS_MorganStanley_ret_5d"], "is_new": true}, {"model_id": "new_h3_NORMAL_LogisticRegression_N8_t0", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 3, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "MS_MorganStanley_ret_1d", "NOC_Northrop_ret_20d", "T10Y2Y_Spread_ret_5d", "Michigan_Sentiment_ret_20d", "MSTR_Bitcoin3_ret_20d", "heston_var_ev_h5"], "is_new": true}, {"model_id": "new_h3_NORMAL_LogisticRegression_N8_t1", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 3, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "VOD_Vodafone_zscore_60d", "MO_AltriaMG_ret_1d", "NOC_Northrop_ret_20d", "IYR_US_REIT2_zscore_60d", "EWQ_France_zscore_60d", "BA_ret_1d"], "is_new": true}, {"model_id": "new_h3_NORMAL_LogisticRegression_N8_t2", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 3, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "SBUX_vol_20d", "LUV_SouthwestAir_ret_5d", "AMD_ret_5d", "XLV_Health_zscore_60d", "EWG_Germany_ret_20d", "NEE_NextEra_ret_20d"], "is_new": true}, {"model_id": "new_h3_NORMAL_LogisticRegression_N8_t3", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 3, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "3M_vol_20d", "XLY_Disc_vol_20d", "EFFR_ret_1d", "US30Y_Rate_ret_20d", "ASX_Australia_vol_20d", "EWQ_France_ret_20d"], "is_new": true}, {"model_id": "new_h3_NORMAL_LogisticRegression_N8_t4", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 3, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "DHR_vol_20d", "CTAS_Cintas_vol_20d", "EWL_Switzerland_vol_20d", "PPL_PPL_ret_1d", "HD_ret_5d", "PAYX_Paychex_zscore_60d"], "is_new": true}, {"model_id": "new_h3_NORMAL_LogisticRegression_N8_t5", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 3, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "gjr_condvar_h1", "DHR_vol_20d", "NEE_NextEra_ret_20d", "SJM_JM_Smucker_ret_5d", "XOM_ret_1d", "HD_ret_1d"], "is_new": true}, {"model_id": "new_h3_NORMAL_LogisticRegression_N8_t6", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 3, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "SJM_JM_Smucker_ret_1d", "NOC_Northrop_ret_20d", "DAX_Germany_zscore_60d", "XLK_Tech_zscore_60d", "EWY_Korea_ret_20d", "MS_MorganStanley_zscore_60d"], "is_new": true}, {"model_id": "new_h3_NORMAL_LogisticRegression_N8_t7", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 3, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "AORD_AUS_zscore_60d", "PPL_PPL_ret_1d", "EOG_EOGResources_vol_20d", "SO_SouthernCo_ret_5d", "CI_Cigna_vol_20d", "XLV_Health_zscore_60d"], "is_new": true}, {"model_id": "new_h3_NORMAL_LogisticRegression_N10_t0", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 3, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "vix_acceleration_1d", "SJM_JM_Smucker_ret_5d", "AMZN_ret_5d", "VOD_Vodafone_zscore_60d", "SLB_Schlumberger_ret_1d", "AORD_AUS_zscore_60d", "WTI_Oil_FRED_zscore_60d", "TGT_Target_zscore_60d"], "is_new": true}, {"model_id": "new_h3_NORMAL_LogisticRegression_N10_t1", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 3, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "XOM_ret_1d", "EWH_HongKong_ret_5d", "EWM_Malaysia_zscore_60d", "MS_MorganStanley_ret_5d", "EQIX_Equinix_ret_5d", "ORCL_zscore_60d", "DIS_vol_20d", "GD_GeneralDynamics_zscore_60d"], "is_new": true}, {"model_id": "new_h3_NORMAL_LogisticRegression_N10_t2", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 3, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "SBUX_ret_5d", "LOW_Lowes_ret_5d", "XLB_Materials_zscore_60d", "EXC_Exelon_zscore_60d", "DE_Deere_vol_20d", "MS_MorganStanley_ret_1d", "SBUX_zscore_60d", "EWM_Malaysia_zscore_60d"], "is_new": true}, {"model_id": "new_h3_NORMAL_LogisticRegression_N10_t3", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 3, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "DIS_vol_20d", "CPB_CampbellSoup_vol_20d", "INTC_ret_1d", "LMT_LockheedMartin_ret_1d", "EWQ_France_ret_20d", "INTC_ret_5d", "AXP_Amex_vol_20d", "CPB_CampbellSoup_ret_20d"], "is_new": true}, {"model_id": "new_h3_NORMAL_LogisticRegression_N10_t4", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 3, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWY_Korea_zscore_60d", "LOW_Lowes_ret_5d", "CPB_CampbellSoup_ret_20d", "CI_Cigna_vol_20d", "vix_acceleration_1d", "MS_MorganStanley_zscore_60d", "Industrial_Production_zscore_60d", "BTI_BritishAmerican_ret_5d"], "is_new": true}, {"model_id": "new_h3_NORMAL_LogisticRegression_N10_t5", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 3, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "HUM_Humana_ret_5d", "VRP_ma5", "IWM_SmallCap_vol_20d", "CPB_CampbellSoup_zscore_60d", "EWM_Malaysia_ret_1d", "EWG_Germany_ret_20d", "US6M_Rate_ret_20d", "NWL_Newell_ret_20d"], "is_new": true}, {"model_id": "new_h3_NORMAL_LogisticRegression_N10_t6", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 3, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "ITT_ITTInc_ret_5d", "DOW_Price_zscore_60d", "heston_var_ev_h5", "US5Y_Rate_ret_5d", "Michigan_Sentiment_ret_20d", "MS_MorganStanley_zscore_60d", "HangSeng_HK_vol_20d", "Retail_Sales_zscore_60d"], "is_new": true}, {"model_id": "new_h3_NORMAL_LogisticRegression_N10_t7", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 3, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "DE_Deere_vol_20d", "spx_momentum_3d", "3M_ret_5d", "PCAR_PaccarInc_ret_5d", "EFFR_ret_1d", "EWA_Australia_zscore_60d", "US3Y_Rate_ret_5d", "Core_PCE_zscore_60d"], "is_new": true}, {"model_id": "new_h3_NORMAL_LogisticRegression_N12_t0", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 3, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "IYM_BasicMaterials_ret_20d", "LUV_SouthwestAir_ret_5d", "BA_ret_1d", "NFCI_ret_5d", "EWL_Switzerland_vol_20d", "Nikkei_Japan_zscore_60d", "US6M_Rate_ret_20d", "vix_acceleration_1d", "SLB_Schlumberger_ret_5d", "JNJ_ret_1d"], "is_new": true}, {"model_id": "new_h3_NORMAL_LogisticRegression_N12_t1", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 3, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "hmm_p_stress", "LOW_Lowes_ret_5d", "LMT_LockheedMartin_vol_20d", "XLY_Disc_vol_20d", "US6M_Rate_ret_20d", "Brent_Oil_FRED_ret_5d", "VVIX_ret_20d", "MSTR_Bitcoin3_ret_20d", "NFCI_ret_5d", "US7Y_Rate_ret_20d"], "is_new": true}, {"model_id": "new_h3_NORMAL_LogisticRegression_N12_t2", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 3, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "DHR_ret_1d", "EWC_Canada_zscore_60d", "MSTR_Bitcoin3_ret_1d", "GILD_Gilead_ret_20d", "HangSeng_HK_vol_20d", "US3Y_Rate_ret_5d", "LLY_zscore_60d", "CPB_CampbellSoup_vol_20d", "HD_zscore_60d", "EWQ_France_ret_20d"], "is_new": true}, {"model_id": "new_h3_NORMAL_LogisticRegression_N12_t3", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 3, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWY_Korea_zscore_60d", "EFFR_ret_1d", "HUM_Humana_ret_5d", "XLF_Fin_vol_20d", "CPB_CampbellSoup_vol_20d", "NVDA_vol_20d", "Nikkei_Japan_zscore_60d", "EWC_Canada_zscore_60d", "Core_PCE_zscore_60d", "SJM_JM_Smucker_ret_1d"], "is_new": true}, {"model_id": "new_h3_NORMAL_LogisticRegression_N12_t4", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 3, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "DIS_vol_20d", "EQR_Equity_ret_1d", "CPB_CampbellSoup_ret_5d", "ES_Evergy_ret_1d", "HangSeng_HK_ret_5d", "DE_Deere_ret_5d", "heston_var_ev_h7", "CPB_CampbellSoup_zscore_60d", "VRP_ma5", "Retail_Sales_zscore_60d"], "is_new": true}, {"model_id": "new_h3_NORMAL_LogisticRegression_N12_t5", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 3, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EXC_Exelon_ret_1d", "PAYX_Paychex_ret_20d", "EWG_Germany_vol_20d", "CPB_CampbellSoup_vol_20d", "PCAR_PaccarInc_ret_5d", "IWM_SmallCap_vol_20d", "CPB_CampbellSoup_ret_5d", "ASX_Australia_ret_5d", "spx_vol_5d", "EQIX_Equinix_ret_5d"], "is_new": true}, {"model_id": "new_h3_NORMAL_LogisticRegression_N12_t6", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 3, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "AMD_ret_5d", "SLB_Schlumberger_ret_1d", "EWG_Germany_vol_20d", "EWQ_France_ret_20d", "EOG_EOGResources_ret_5d", "QQQ_vol_20d", "CPB_CampbellSoup_ret_5d", "AMZN_ret_5d", "HangSeng_HK_ret_1d", "INTC_ret_1d"], "is_new": true}, {"model_id": "new_h3_NORMAL_LogisticRegression_N12_t7", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 3, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "INTC_ret_5d", "hmm_p_stress", "Nikkei_Japan_vol_20d", "EWG_Germany_ret_20d", "HUM_Humana_ret_5d", "EWM_Malaysia_ret_1d", "BTI_BritishAmerican_ret_20d", "ORCL_zscore_60d", "Brent_Oil_FRED_ret_5d", "XLB_Materials_zscore_60d"], "is_new": true}, {"model_id": "new_h3_NORMAL_LogisticRegression_N15_t0", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 3, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWC_Canada_zscore_60d", "LMT_LockheedMartin_ret_1d", "EXC_Exelon_zscore_60d", "PAYX_Paychex_vol_20d", "XOM_ret_20d", "CI_Cigna_vol_20d", "LOW_Lowes_ret_20d", "EWJ_Japan_vol_20d", "EWS_Singapore_ret_5d", "heston_var_ev_h3", "heston_ev_h3", "SLB_Schlumberger_ret_1d", "ENB_EnbridgeInc_ret_1d"], "is_new": true}, {"model_id": "new_h3_NORMAL_LogisticRegression_N15_t1", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 3, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "NFCI_ret_5d", "TXN_vol_20d", "US3Y_Rate_ret_5d", "DHR_vol_20d", "DIS_vol_20d", "ASX_Australia_ret_5d", "MS_MorganStanley_zscore_60d", "TM_Telephone_vol_20d", "DE_Deere_vol_20d", "US3M_Rate_zscore_60d", "heston_var_ev_h7", "VRP_ma5", "EXC_Exelon_ret_1d"], "is_new": true}, {"model_id": "new_h3_NORMAL_LogisticRegression_N15_t2", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 3, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "LLY_zscore_60d", "PFE_ret_1d", "Core_PCE_zscore_60d", "US1Y_Rate_ret_5d", "NOC_Northrop_ret_20d", "US3Y_Rate_ret_5d", "CPB_CampbellSoup_ret_5d", "US3M_Rate_vol_20d", "spx_vol_5d", "heston_var_ev_h5", "SO_SouthernCo_ret_5d", "CI_Cigna_vol_20d", "Nikkei_Japan_vol_20d"], "is_new": true}, {"model_id": "new_h3_NORMAL_LogisticRegression_N15_t3", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 3, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "VVIX_ret_20d", "XLB_Materials_zscore_60d", "EWC_Canada_zscore_60d", "CPB_CampbellSoup_zscore_60d", "HangSeng_HK_ret_5d", "ITT_ITTInc_ret_5d", "EWY_Korea_ret_20d", "NVDA_vol_20d", "Industrial_Production_zscore_60d", "CPB_CampbellSoup_ret_5d", "EWG_Germany_ret_20d", "EWL_Switzerland_vol_20d", "SPY_zscore_60d"], "is_new": true}, {"model_id": "new_h3_NORMAL_LogisticRegression_N15_t4", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 3, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "PLD_Prologis_ret_5d", "HD_zscore_60d", "MSTR_Bitcoin3_ret_5d", "CCI_CrownCastle_vol_20d", "Nikkei_Japan_zscore_60d", "CPB_CampbellSoup_ret_5d", "Nikkei_Japan_vol_20d", "AMD_ret_1d", "INTC_ret_5d", "GE_ret_1d", "EWQ_France_zscore_60d", "EWM_Malaysia_ret_1d", "PAYX_Paychex_zscore_60d"], "is_new": true}, {"model_id": "new_h3_NORMAL_LogisticRegression_N15_t5", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 3, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "MSTR_Bitcoin3_ret_20d", "PPL_PPL_ret_1d", "CCI_CrownCastle_vol_20d", "DAX_Germany_zscore_60d", "SJM_JM_Smucker_ret_1d", "EXC_Exelon_zscore_60d", "SBUX_ret_5d", "3M_vol_20d", "NOC_Northrop_ret_20d", "XLF_Fin_vol_20d", "MRK_Merck_zscore_60d", "EWQ_France_zscore_60d", "Brent_Oil_FRED_ret_5d"], "is_new": true}, {"model_id": "new_h3_NORMAL_LogisticRegression_N15_t6", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 3, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "3M_ret_5d", "spx_vol_5d", "MS_MorganStanley_ret_1d", "XOM_ret_1d", "PAYX_Paychex_zscore_60d", "DHR_ret_1d", "VVIX_ret_20d", "Nikkei_Japan_vol_20d", "PAYX_Paychex_vol_20d", "DHR_vol_20d", "BA_ret_1d", "GE_ret_1d", "CLX_Clorox_vol_20d"], "is_new": true}, {"model_id": "new_h3_NORMAL_LogisticRegression_N15_t7", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 3, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "BTI_BritishAmerican_ret_20d", "XLV_Health_zscore_60d", "CPB_CampbellSoup_zscore_60d", "heston_var_ev_h7", "MSTR_Bitcoin3_ret_1d", "PAYX_Paychex_ret_20d", "US1Y_Rate_ret_20d", "gjr_condvar_h1", "PAYX_Paychex_zscore_60d", "INTC_ret_1d", "US1Y_Rate_ret_5d", "MSTR_Bitcoin3_ret_5d", "EXC_Exelon_zscore_60d"], "is_new": true}, {"model_id": "new_h3_NORMAL_LogisticRegression_N20_t0", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 3, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "XLF_Fin_vol_20d", "DHR_vol_20d", "heston_var_ev_h3", "LUV_SouthwestAir_ret_5d", "LLY_zscore_60d", "EWY_Korea_ret_20d", "Core_PCE_zscore_60d", "PAYX_Paychex_zscore_60d", "IYR_US_REIT2_zscore_60d", "IWM_SmallCap_vol_20d", "3M_ret_5d", "EXC_Exelon_zscore_60d", "MS_MorganStanley_ret_5d", "CPB_CampbellSoup_vol_20d", "ORCL_vol_20d", "CTAS_Cintas_vol_20d", "HUM_Humana_ret_5d", "EWM_Malaysia_zscore_60d"], "is_new": true}, {"model_id": "new_h3_NORMAL_LogisticRegression_N20_t1", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 3, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "XLB_Materials_zscore_60d", "PLD_Prologis_ret_5d", "DAX_Germany_zscore_60d", "MSTR_Bitcoin3_ret_1d", "Retail_Sales_zscore_60d", "Industrial_Production_zscore_60d", "HD_ret_20d", "LLY_zscore_60d", "ORCL_zscore_60d", "NVDA_vol_20d", "VOD_Vodafone_zscore_60d", "gjr_condvar_h1", "BDX_Becton_Dickinson_ret_20d", "US6M_Rate_ret_20d", "HUM_Humana_ret_5d", "CLX_Clorox_vol_20d", "EXC_Exelon_zscore_60d", "DE_Deere_ret_5d"], "is_new": true}, {"model_id": "new_h3_NORMAL_LogisticRegression_N20_t2", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 3, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWC_Canada_zscore_60d", "NEE_NextEra_ret_20d", "BDX_Becton_Dickinson_ret_20d", "EWL_Switzerland_vol_20d", "SLB_Schlumberger_ret_5d", "IYM_BasicMaterials_ret_20d", "heston_var_ev_h7", "ITT_ITTInc_ret_5d", "EMR_Emerson_ret_20d", "SLB_Schlumberger_ret_1d", "T10Y2Y_Spread_ret_5d", "ES_Evergy_ret_1d", "EWA_Australia_ret_1d", "Core_CPI_zscore_60d", "EWH_HongKong_ret_5d", "DOW_Price_zscore_60d", "XLY_Disc_vol_20d", "Nikkei_Japan_vol_20d"], "is_new": true}, {"model_id": "new_h3_NORMAL_LogisticRegression_N20_t3", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 3, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "LOW_Lowes_ret_20d", "TM_Telephone_ret_1d", "vix_mean_abs_ret_5d", "LUV_SouthwestAir_ret_5d", "XLY_Disc_vol_20d", "US6M_Rate_ret_20d", "CPB_CampbellSoup_zscore_60d", "XLF_Fin_vol_20d", "PG_ret_20d", "PLD_Prologis_ret_5d", "NEE_NextEra_ret_20d", "EWY_Korea_ret_20d", "SO_SouthernCo_ret_5d", "LOW_Lowes_ret_5d", "MS_MorganStanley_ret_5d", "spx_momentum_3d", "US5Y_Rate_ret_5d", "DIS_vol_20d"], "is_new": true}, {"model_id": "new_h3_NORMAL_LogisticRegression_N20_t4", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 3, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "SBUX_vol_20d", "NEE_NextEra_ret_20d", "EWC_Canada_zscore_60d", "NVDA_vol_20d", "Michigan_Sentiment_ret_20d", "EQIX_Equinix_ret_5d", "SJM_JM_Smucker_ret_5d", "SJM_JM_Smucker_ret_1d", "CPB_CampbellSoup_ret_20d", "Brent_Oil_FRED_ret_5d", "PG_ret_20d", "Core_CPI_zscore_60d", "ORCL_vol_20d", "MO_AltriaMG_ret_1d", "EWJ_Japan_vol_20d", "heston_var_ev_h3", "JNJ_ret_1d", "VVIX_ret_20d"], "is_new": true}, {"model_id": "new_h3_NORMAL_LogisticRegression_N20_t5", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 3, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "MO_AltriaMG_ret_1d", "AXP_Amex_vol_20d", "NWL_Newell_ret_20d", "vix_acceleration_1d", "US3M_Rate_zscore_60d", "T10Y2Y_Spread_ret_5d", "XLV_Health_zscore_60d", "Industrial_Production_zscore_60d", "SJM_JM_Smucker_ret_5d", "spx_momentum_3d", "DAX_Germany_zscore_60d", "XLF_Fin_vol_20d", "FedFunds_zscore_60d", "TM_Telephone_ret_1d", "NFCI_ret_5d", "EWQ_France_ret_20d", "BLK_BlackRock_zscore_60d", "US3Y_Rate_ret_5d"], "is_new": true}, {"model_id": "new_h3_NORMAL_LogisticRegression_N20_t6", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 3, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "IWM_SmallCap_vol_20d", "XOM_ret_20d", "GILD_Gilead_ret_20d", "DAX_Germany_vol_20d", "PAYX_Paychex_vol_20d", "US6M_Rate_ret_20d", "EWG_Germany_vol_20d", "XLV_Health_zscore_60d", "HangSeng_HK_ret_1d", "EWL_Switzerland_zscore_60d", "CI_Cigna_vol_20d", "IBEX_Spain_ret_20d", "US3M_Rate_vol_20d", "Core_PCE_zscore_60d", "FedFunds_zscore_60d", "EFFR_vol_20d", "Brent_Oil_FRED_ret_20d", "LOW_Lowes_ret_20d"], "is_new": true}, {"model_id": "new_h3_NORMAL_LogisticRegression_N20_t7", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 3, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "PG_ret_20d", "MS_MorganStanley_ret_1d", "DIS_vol_20d", "IBEX_Spain_ret_20d", "DHR_vol_20d", "T_ret_1d", "LOW_Lowes_ret_5d", "SCHW_Schwab_ret_5d", "EXC_Exelon_zscore_60d", "ASX_Australia_ret_5d", "AMT_AmericanTower_ret_1d", "MO_AltriaMG_ret_1d", "EFFR_vol_20d", "vix_mean_abs_ret_5d", "SBUX_vol_20d", "heston_ev_h3", "PLD_Prologis_ret_5d", "EWG_Germany_ret_20d"], "is_new": true}, {"model_id": "new_h3_NORMAL_LogisticRegression_N25_t0", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 3, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "MSTR_Bitcoin3_ret_20d", "EWM_Malaysia_zscore_60d", "AXP_Amex_vol_20d", "PAYX_Paychex_zscore_60d", "PAYX_Paychex_ret_20d", "US3Y_Rate_ret_5d", "XLF_Fin_vol_20d", "SBUX_zscore_60d", "ITT_ITTInc_ret_5d", "Brent_Oil_FRED_ret_5d", "3M_ret_5d", "US7Y_Rate_ret_20d", "AORD_AUS_zscore_60d", "MS_MorganStanley_zscore_60d", "LMT_LockheedMartin_vol_20d", "CPB_CampbellSoup_ret_5d", "HD_ret_20d", "AMT_AmericanTower_ret_1d", "SBUX_vol_20d", "EOG_EOGResources_ret_5d", "EWM_Malaysia_vol_20d", "HD_ret_1d", "NEE_NextEra_ret_20d"], "is_new": true}, {"model_id": "new_h3_NORMAL_LogisticRegression_N25_t1", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 3, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "Core_PCE_zscore_60d", "BTI_BritishAmerican_ret_20d", "SJM_JM_Smucker_ret_1d", "AXP_Amex_vol_20d", "SO_SouthernCo_ret_5d", "EFFR_ret_1d", "US3M_Rate_vol_20d", "DOW_Price_zscore_60d", "DHR_ret_1d", "BA_ret_1d", "VOD_Vodafone_zscore_60d", "DHR_vol_20d", "INTC_ret_5d", "CLX_Clorox_vol_20d", "EWH_HongKong_ret_5d", "AORD_AUS_zscore_60d", "3M_vol_20d", "EQIX_Equinix_ret_5d", "US7Y_Rate_ret_20d", "LOW_Lowes_ret_20d", "AXP_Amex_ret_20d", "vix_acceleration_1d", "CMCSA_ret_1d"], "is_new": true}, {"model_id": "new_h3_NORMAL_LogisticRegression_N25_t2", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 3, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EQIX_Equinix_ret_5d", "XLV_Health_zscore_60d", "US3Y_Rate_ret_5d", "BDX_Becton_Dickinson_ret_20d", "LLY_zscore_60d", "MO_AltriaMG_ret_1d", "ITT_ITTInc_ret_5d", "DAX_Germany_zscore_60d", "EWQ_France_zscore_60d", "SBUX_ret_5d", "SLB_Schlumberger_ret_1d", "EWA_Australia_ret_1d", "XLY_Disc_vol_20d", "AORD_AUS_zscore_60d", "EWY_Korea_zscore_60d", "XLK_Tech_zscore_60d", "SCHW_Schwab_ret_5d", "spx_abs_ret_max_5d", "Retail_Sales_zscore_60d", "HD_zscore_60d", "INTC_ret_1d", "MSTR_Bitcoin3_ret_1d", "TXN_vol_20d"], "is_new": true}, {"model_id": "new_h3_NORMAL_LogisticRegression_N25_t3", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 3, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWM_Malaysia_zscore_60d", "EWC_Canada_zscore_60d", "EWH_HongKong_ret_5d", "EWS_Singapore_ret_5d", "heston_ev_h3", "US1Y_Rate_ret_5d", "TXN_vol_20d", "HangSeng_HK_vol_20d", "NOC_Northrop_ret_20d", "JNJ_ret_1d", "BTI_BritishAmerican_ret_20d", "US3Y_Rate_ret_5d", "AVB_AvalonBay_zscore_60d", "spx_vol_5d", "HD_zscore_60d", "US30Y_Rate_ret_20d", "EWG_Germany_ret_20d", "BA_ret_1d", "XLK_Tech_zscore_60d", "DHR_ret_1d", "HUM_Humana_ret_5d", "CLX_Clorox_vol_20d", "XLY_Disc_vol_20d"], "is_new": true}, {"model_id": "new_h3_NORMAL_LogisticRegression_N25_t4", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 3, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "Michigan_Sentiment_ret_20d", "spx_vol_5d", "EXC_Exelon_ret_1d", "FedFunds_zscore_60d", "US6M_Rate_ret_20d", "AVB_AvalonBay_zscore_60d", "EWG_Germany_ret_20d", "LUV_SouthwestAir_ret_5d", "heston_var_ev_h3", "US3M_Rate_vol_20d", "EWS_Singapore_ret_5d", "HD_zscore_60d", "Core_PCE_zscore_60d", "SBUX_vol_20d", "SBUX_ret_5d", "EWA_Australia_zscore_60d", "EWG_Germany_vol_20d", "HD_ret_20d", "EWM_Malaysia_zscore_60d", "hmm_p_stress", "PCAR_PaccarInc_ret_5d", "CTAS_Cintas_vol_20d", "T_ret_1d"], "is_new": true}, {"model_id": "new_h3_NORMAL_LogisticRegression_N25_t5", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 3, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "CMCSA_ret_1d", "MSTR_Bitcoin3_ret_5d", "AMGN_Amgen_ret_1d", "ORCL_vol_20d", "EWA_Australia_ret_1d", "US3M_Rate_zscore_60d", "ENB_EnbridgeInc_ret_1d", "EWL_Switzerland_zscore_60d", "BTI_BritishAmerican_ret_5d", "TED_Spread_zscore_60d", "MSTR_Bitcoin3_ret_1d", "PG_ret_20d", "T10Y2Y_Spread_ret_5d", "Core_CPI_zscore_60d", "CI_Cigna_vol_20d", "AMZN_ret_5d", "TXN_vol_20d", "BLK_BlackRock_zscore_60d", "EFFR_ret_1d", "EWQ_France_ret_20d", "TM_Telephone_vol_20d", "US1Y_Rate_ret_5d", "EWQ_France_zscore_60d"], "is_new": true}, {"model_id": "new_h3_NORMAL_LogisticRegression_N25_t6", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 3, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "vix_mean_abs_ret_5d", "LOW_Lowes_ret_20d", "PAYX_Paychex_zscore_60d", "XLF_Fin_vol_20d", "INTC_ret_5d", "EWQ_France_zscore_60d", "vix_acceleration_1d", "NWL_Newell_ret_20d", "IBEX_Spain_ret_20d", "GE_ret_1d", "BTI_BritishAmerican_ret_20d", "BTI_BritishAmerican_ret_5d", "spx_momentum_3d", "EWA_Australia_ret_1d", "LMT_LockheedMartin_ret_1d", "HD_zscore_60d", "spx_vol_5d", "ES_Evergy_ret_1d", "T10Y2Y_Spread_ret_5d", "MS_MorganStanley_ret_5d", "EWY_Korea_ret_20d", "US3M_Rate_vol_20d", "VVIX_ret_20d"], "is_new": true}, {"model_id": "new_h3_NORMAL_LogisticRegression_N25_t7", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 3, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWM_Malaysia_zscore_60d", "SJM_JM_Smucker_ret_5d", "FedFunds_zscore_60d", "XOM_ret_1d", "DHR_vol_20d", "BDX_Becton_Dickinson_ret_20d", "EWM_Malaysia_ret_1d", "Brent_Oil_FRED_ret_20d", "QQQ_vol_20d", "heston_ev_h3", "LLY_zscore_60d", "spx_momentum_3d", "Brent_Oil_FRED_ret_5d", "EXC_Exelon_zscore_60d", "ITT_ITTInc_ret_5d", "AMD_ret_1d", "XLF_Fin_vol_20d", "AXP_Amex_ret_20d", "WTI_Oil_FRED_zscore_60d", "AMZN_ret_5d", "EWA_Australia_ret_1d", "LUV_SouthwestAir_ret_5d", "Retail_Sales_zscore_60d"], "is_new": true}, {"model_id": "new_h3_NORMAL_LogisticRegression_N30_t0", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 3, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWQ_France_ret_20d", "US5Y_Rate_ret_5d", "heston_var_ev_h3", "CLX_Clorox_vol_20d", "AMZN_ret_5d", "MSTR_Bitcoin3_ret_5d", "SBUX_ret_5d", "EWM_Malaysia_ret_1d", "EWL_Switzerland_zscore_60d", "EWJ_Japan_vol_20d", "T10Y2Y_Spread_ret_5d", "ES_Evergy_ret_1d", "PAYX_Paychex_ret_20d", "hmm_p_stress", "INTC_ret_5d", "EWL_Switzerland_vol_20d", "HUM_Humana_ret_5d", "TM_Telephone_vol_20d", "BA_ret_1d", "EQIX_Equinix_ret_5d", "HangSeng_HK_ret_1d", "vix_acceleration_1d", "MS_MorganStanley_ret_1d", "MS_MorganStanley_ret_5d", "gjr_condvar_h1", "EOG_EOGResources_ret_5d", "vix_mean_abs_ret_5d", "US1Y_Rate_ret_5d"], "is_new": true}, {"model_id": "new_h3_NORMAL_LogisticRegression_N30_t1", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 3, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "AMZN_ret_5d", "Brent_Oil_FRED_ret_20d", "3M_ret_5d", "CTAS_Cintas_vol_20d", "heston_ev_h3", "LUV_SouthwestAir_ret_5d", "IBEX_Spain_ret_20d", "HangSeng_HK_ret_5d", "HangSeng_HK_vol_20d", "T_ret_1d", "heston_var_ev_h7", "SBUX_zscore_60d", "VVIX_ret_20d", "DHR_vol_20d", "3M_vol_20d", "PAYX_Paychex_vol_20d", "GE_ret_1d", "BDX_Becton_Dickinson_ret_20d", "hmm_p_stress", "CPB_CampbellSoup_vol_20d", "spx_abs_ret_max_5d", "PLD_Prologis_ret_5d", "AMD_ret_5d", "EWA_Australia_ret_1d", "DE_Deere_vol_20d", "EOG_EOGResources_ret_5d", "XOM_ret_1d", "PAYX_Paychex_zscore_60d"], "is_new": true}, {"model_id": "new_h3_NORMAL_LogisticRegression_N30_t2", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 3, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "LLY_zscore_60d", "EWY_Korea_zscore_60d", "XOM_ret_20d", "VVIX_ret_20d", "TED_Spread_zscore_60d", "EWQ_France_zscore_60d", "US3Y_Rate_ret_5d", "Brent_Oil_FRED_ret_5d", "AMZN_ret_5d", "EWH_HongKong_ret_5d", "EQIX_Equinix_ret_5d", "3M_vol_20d", "GILD_Gilead_ret_20d", "CPB_CampbellSoup_zscore_60d", "BA_ret_1d", "EWM_Malaysia_vol_20d", "SBUX_vol_20d", "DE_Deere_ret_5d", "SLB_Schlumberger_ret_5d", "BTI_BritishAmerican_ret_5d", "BDX_Becton_Dickinson_ret_20d", "DOW_Price_zscore_60d", "PCAR_PaccarInc_ret_5d", "EOG_EOGResources_vol_20d", "JNJ_ret_1d", "PAYX_Paychex_vol_20d", "IWM_SmallCap_vol_20d", "LOW_Lowes_ret_20d"], "is_new": true}, {"model_id": "new_h3_NORMAL_LogisticRegression_N30_t3", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 3, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "PG_ret_20d", "BDX_Becton_Dickinson_ret_20d", "heston_var_ev_h5", "vix_acceleration_1d", "SCHW_Schwab_ret_5d", "XLF_Fin_vol_20d", "Nikkei_Japan_vol_20d", "EWJ_Japan_vol_20d", "EWH_HongKong_ret_5d", "AMT_AmericanTower_ret_1d", "EQR_Equity_ret_1d", "EFFR_vol_20d", "LMT_LockheedMartin_ret_1d", "CPB_CampbellSoup_vol_20d", "PLD_Prologis_ret_5d", "MS_MorganStanley_ret_1d", "heston_var_ev_h7", "ORCL_vol_20d", "TGT_Target_zscore_60d", "EOG_EOGResources_vol_20d", "Core_PCE_zscore_60d", "ENB_EnbridgeInc_ret_1d", "SBUX_zscore_60d", "HD_ret_5d", "NOC_Northrop_ret_20d", "EMR_Emerson_ret_20d", "CPB_CampbellSoup_ret_5d", "heston_var_ev_h3"], "is_new": true}, {"model_id": "new_h3_NORMAL_LogisticRegression_N30_t4", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 3, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "US30Y_Rate_ret_20d", "EQIX_Equinix_ret_5d", "heston_var_ev_h3", "XOM_ret_1d", "MO_AltriaMG_ret_1d", "MS_MorganStanley_zscore_60d", "LMT_LockheedMartin_ret_1d", "hmm_p_stress", "EFFR_vol_20d", "BTI_BritishAmerican_ret_20d", "US6M_Rate_ret_20d", "EOG_EOGResources_ret_5d", "WTI_Oil_FRED_zscore_60d", "IWM_SmallCap_vol_20d", "spx_momentum_3d", "ASX_Australia_ret_5d", "US7Y_Rate_ret_20d", "US3M_Rate_zscore_60d", "HangSeng_HK_ret_5d", "HangSeng_HK_ret_1d", "Brent_Oil_FRED_ret_5d", "VVIX_ret_20d", "SJM_JM_Smucker_ret_1d", "LUV_SouthwestAir_ret_5d", "EWH_HongKong_ret_5d", "ORCL_zscore_60d", "Nikkei_Japan_vol_20d", "EWC_Canada_zscore_60d"], "is_new": true}, {"model_id": "new_h3_NORMAL_LogisticRegression_N30_t5", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 3, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "HD_zscore_60d", "SLB_Schlumberger_ret_5d", "LMT_LockheedMartin_vol_20d", "TXN_vol_20d", "Nikkei_Japan_vol_20d", "IBEX_Spain_ret_20d", "SJM_JM_Smucker_ret_1d", "ORCL_vol_20d", "NOC_Northrop_ret_20d", "US7Y_Rate_ret_20d", "LOW_Lowes_ret_20d", "ITT_ITTInc_ret_5d", "heston_var_ev_h7", "SBUX_vol_20d", "3M_vol_20d", "XOM_ret_20d", "Core_PCE_zscore_60d", "US30Y_Rate_ret_20d", "NEE_NextEra_ret_20d", "LUV_SouthwestAir_ret_5d", "BA_ret_1d", "MS_MorganStanley_zscore_60d", "PG_ret_20d", "US6M_Rate_ret_20d", "NFCI_ret_5d", "HD_ret_1d", "Industrial_Production_zscore_60d", "BTI_BritishAmerican_ret_5d"], "is_new": true}, {"model_id": "new_h3_NORMAL_LogisticRegression_N30_t6", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 3, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "XLV_Health_zscore_60d", "DHR_ret_1d", "IYM_BasicMaterials_ret_20d", "NFCI_ret_5d", "SJM_JM_Smucker_ret_5d", "PCAR_PaccarInc_ret_5d", "3M_ret_5d", "IBEX_Spain_ret_20d", "ENB_EnbridgeInc_ret_1d", "NWL_Newell_ret_20d", "FedFunds_zscore_60d", "MSTR_Bitcoin3_ret_1d", "XLK_Tech_zscore_60d", "SJM_JM_Smucker_ret_1d", "LOW_Lowes_ret_20d", "AXP_Amex_ret_20d", "TXN_vol_20d", "AXP_Amex_vol_20d", "HangSeng_HK_ret_1d", "MS_MorganStanley_zscore_60d", "NEE_NextEra_ret_20d", "3M_vol_20d", "MS_MorganStanley_ret_1d", "EWL_Switzerland_zscore_60d", "SLB_Schlumberger_ret_1d", "EQR_Equity_ret_1d", "US5Y_Rate_ret_5d", "MRK_Merck_zscore_60d"], "is_new": true}, {"model_id": "new_h3_NORMAL_LogisticRegression_N30_t7", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 3, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "DHR_vol_20d", "PG_ret_20d", "IYM_BasicMaterials_ret_20d", "3M_ret_5d", "gjr_condvar_h1", "heston_var_ev_h3", "EMR_Emerson_ret_20d", "AORD_AUS_zscore_60d", "hmm_p_stress", "XLV_Health_zscore_60d", "EFFR_vol_20d", "EQR_Equity_ret_1d", "IYR_US_REIT2_zscore_60d", "EOG_EOGResources_vol_20d", "CTAS_Cintas_vol_20d", "ES_Evergy_ret_1d", "EWM_Malaysia_ret_1d", "US30Y_Rate_ret_20d", "AMD_ret_5d", "XLY_Disc_vol_20d", "US6M_Rate_ret_20d", "HD_ret_20d", "FedFunds_zscore_60d", "MSTR_Bitcoin3_ret_20d", "DE_Deere_vol_20d", "SCHW_Schwab_ret_5d", "MSTR_Bitcoin3_ret_1d", "EWS_Singapore_ret_5d"], "is_new": true}, {"model_id": "new_h3_STRESS_XGBoost_N5_t0", "algo": "XGBoost", "regime": "STRESS", "horizon": 3, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "FedFunds_zscore_60d", "VRP_ma5", "PPL_PPL_ret_1d"], "is_new": true}, {"model_id": "new_h3_STRESS_XGBoost_N5_t1", "algo": "XGBoost", "regime": "STRESS", "horizon": 3, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "DE_Deere_ret_5d", "VVIX_ret_20d", "IYM_BasicMaterials_ret_20d"], "is_new": true}, {"model_id": "new_h3_STRESS_XGBoost_N5_t2", "algo": "XGBoost", "regime": "STRESS", "horizon": 3, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWM_Malaysia_ret_1d", "gjr_condvar_h1", "EWM_Malaysia_vol_20d"], "is_new": true}, {"model_id": "new_h3_STRESS_XGBoost_N5_t3", "algo": "XGBoost", "regime": "STRESS", "horizon": 3, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "VRP_ma5", "LLY_zscore_60d", "EWM_Malaysia_ret_1d"], "is_new": true}, {"model_id": "new_h3_STRESS_XGBoost_N5_t4", "algo": "XGBoost", "regime": "STRESS", "horizon": 3, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "T_ret_1d", "QQQ_vol_20d", "hmm_p_stress"], "is_new": true}, {"model_id": "new_h3_STRESS_XGBoost_N5_t5", "algo": "XGBoost", "regime": "STRESS", "horizon": 3, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "3M_vol_20d", "US3Y_Rate_ret_5d", "AXP_Amex_ret_20d"], "is_new": true}, {"model_id": "new_h3_STRESS_XGBoost_N5_t6", "algo": "XGBoost", "regime": "STRESS", "horizon": 3, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "US5Y_Rate_ret_5d", "PFE_ret_1d", "EWY_Korea_ret_20d"], "is_new": true}, {"model_id": "new_h3_STRESS_XGBoost_N5_t7", "algo": "XGBoost", "regime": "STRESS", "horizon": 3, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "XLF_Fin_vol_20d", "NWL_Newell_ret_20d", "EWQ_France_zscore_60d"], "is_new": true}, {"model_id": "new_h3_STRESS_XGBoost_N8_t0", "algo": "XGBoost", "regime": "STRESS", "horizon": 3, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "BA_ret_1d", "INTC_ret_1d", "vix_mean_abs_ret_5d", "3M_ret_5d", "HD_ret_20d", "ITT_ITTInc_ret_5d"], "is_new": true}, {"model_id": "new_h3_STRESS_XGBoost_N8_t1", "algo": "XGBoost", "regime": "STRESS", "horizon": 3, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "hmm_p_stress", "GD_GeneralDynamics_zscore_60d", "BLK_BlackRock_zscore_60d", "AORD_AUS_zscore_60d", "LOW_Lowes_ret_5d", "XLV_Health_zscore_60d"], "is_new": true}, {"model_id": "new_h3_STRESS_XGBoost_N8_t2", "algo": "XGBoost", "regime": "STRESS", "horizon": 3, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "HD_zscore_60d", "LOW_Lowes_ret_20d", "WTI_Oil_FRED_zscore_60d", "GE_ret_1d", "EOG_EOGResources_ret_5d", "CLX_Clorox_vol_20d"], "is_new": true}, {"model_id": "new_h3_STRESS_XGBoost_N8_t3", "algo": "XGBoost", "regime": "STRESS", "horizon": 3, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "SLB_Schlumberger_ret_5d", "HUM_Humana_ret_5d", "CLX_Clorox_vol_20d", "SO_SouthernCo_ret_5d", "ES_Evergy_ret_1d", "TXN_vol_20d"], "is_new": true}, {"model_id": "new_h3_STRESS_XGBoost_N8_t4", "algo": "XGBoost", "regime": "STRESS", "horizon": 3, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "US3M_Rate_zscore_60d", "ASX_Australia_vol_20d", "PLD_Prologis_ret_5d", "T_ret_1d", "PAYX_Paychex_ret_20d", "EOG_EOGResources_vol_20d"], "is_new": true}, {"model_id": "new_h3_STRESS_XGBoost_N8_t5", "algo": "XGBoost", "regime": "STRESS", "horizon": 3, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "IBEX_Spain_ret_20d", "Core_PCE_zscore_60d", "IYR_US_REIT2_zscore_60d", "XLF_Fin_vol_20d", "AXP_Amex_vol_20d", "US3Y_Rate_ret_5d"], "is_new": true}, {"model_id": "new_h3_STRESS_XGBoost_N8_t6", "algo": "XGBoost", "regime": "STRESS", "horizon": 3, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "PLD_Prologis_ret_5d", "3M_ret_5d", "PAYX_Paychex_ret_20d", "US6M_Rate_ret_20d", "MSTR_Bitcoin3_ret_20d", "spx_momentum_3d"], "is_new": true}, {"model_id": "new_h3_STRESS_XGBoost_N8_t7", "algo": "XGBoost", "regime": "STRESS", "horizon": 3, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "VOD_Vodafone_zscore_60d", "GILD_Gilead_ret_20d", "CTAS_Cintas_vol_20d", "EWJ_Japan_vol_20d", "LUV_SouthwestAir_ret_5d", "CPB_CampbellSoup_ret_5d"], "is_new": true}, {"model_id": "new_h3_STRESS_XGBoost_N10_t0", "algo": "XGBoost", "regime": "STRESS", "horizon": 3, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "WTI_Oil_FRED_zscore_60d", "INTC_ret_1d", "LOW_Lowes_ret_5d", "VOD_Vodafone_zscore_60d", "LUV_SouthwestAir_ret_5d", "TM_Telephone_ret_1d", "EFFR_vol_20d", "SBUX_vol_20d"], "is_new": true}, {"model_id": "new_h3_STRESS_XGBoost_N10_t1", "algo": "XGBoost", "regime": "STRESS", "horizon": 3, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "DE_Deere_vol_20d", "NOC_Northrop_ret_20d", "WTI_Oil_FRED_zscore_60d", "spx_momentum_3d", "CTAS_Cintas_vol_20d", "MRK_Merck_zscore_60d", "MS_MorganStanley_ret_1d", "US3M_Rate_vol_20d"], "is_new": true}, {"model_id": "new_h3_STRESS_XGBoost_N10_t2", "algo": "XGBoost", "regime": "STRESS", "horizon": 3, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "AMT_AmericanTower_ret_1d", "TED_Spread_vol_20d", "heston_var_ev_h7", "EQIX_Equinix_ret_5d", "SJM_JM_Smucker_ret_1d", "NWL_Newell_ret_20d", "DAX_Germany_vol_20d", "AMGN_Amgen_ret_1d"], "is_new": true}, {"model_id": "new_h3_STRESS_XGBoost_N10_t3", "algo": "XGBoost", "regime": "STRESS", "horizon": 3, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "Brent_Oil_FRED_ret_5d", "spx_momentum_3d", "DHR_ret_1d", "AMZN_ret_5d", "XOM_ret_1d", "EXC_Exelon_zscore_60d", "LUV_SouthwestAir_ret_5d", "VOD_Vodafone_zscore_60d"], "is_new": true}, {"model_id": "new_h3_STRESS_XGBoost_N10_t4", "algo": "XGBoost", "regime": "STRESS", "horizon": 3, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "PPL_PPL_ret_1d", "EWG_Germany_vol_20d", "EXC_Exelon_zscore_60d", "WTI_Oil_FRED_zscore_60d", "EWA_Australia_zscore_60d", "US30Y_Rate_ret_20d", "LLY_zscore_60d", "heston_ev_h3"], "is_new": true}, {"model_id": "new_h3_STRESS_XGBoost_N10_t5", "algo": "XGBoost", "regime": "STRESS", "horizon": 3, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "Retail_Sales_zscore_60d", "LOW_Lowes_ret_20d", "US3M_Rate_zscore_60d", "SO_SouthernCo_ret_5d", "GD_GeneralDynamics_zscore_60d", "VRP_ma5", "CI_Cigna_vol_20d", "EWS_Singapore_ret_5d"], "is_new": true}, {"model_id": "new_h3_STRESS_XGBoost_N10_t6", "algo": "XGBoost", "regime": "STRESS", "horizon": 3, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "CTAS_Cintas_vol_20d", "US30Y_Rate_ret_20d", "SPY_zscore_60d", "CLX_Clorox_vol_20d", "gjr_condvar_h1", "spx_vol_5d", "Core_PCE_zscore_60d", "EWM_Malaysia_zscore_60d"], "is_new": true}, {"model_id": "new_h3_STRESS_XGBoost_N10_t7", "algo": "XGBoost", "regime": "STRESS", "horizon": 3, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWH_HongKong_ret_5d", "EWG_Germany_ret_20d", "CPB_CampbellSoup_ret_20d", "GD_GeneralDynamics_zscore_60d", "EWQ_France_ret_20d", "ENB_EnbridgeInc_ret_1d", "GILD_Gilead_ret_20d", "CCI_CrownCastle_vol_20d"], "is_new": true}, {"model_id": "new_h3_STRESS_XGBoost_N12_t0", "algo": "XGBoost", "regime": "STRESS", "horizon": 3, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "CPB_CampbellSoup_ret_5d", "DIS_vol_20d", "SBUX_zscore_60d", "SJM_JM_Smucker_ret_5d", "SBUX_vol_20d", "XLB_Materials_zscore_60d", "NOC_Northrop_ret_20d", "EWQ_France_zscore_60d", "EWS_Singapore_ret_5d", "SLB_Schlumberger_ret_5d"], "is_new": true}, {"model_id": "new_h3_STRESS_XGBoost_N12_t1", "algo": "XGBoost", "regime": "STRESS", "horizon": 3, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EOG_EOGResources_vol_20d", "AXP_Amex_vol_20d", "SLB_Schlumberger_ret_1d", "DHR_vol_20d", "EQR_Equity_ret_1d", "BA_ret_1d", "HangSeng_HK_vol_20d", "HD_ret_20d", "SPY_zscore_60d", "HUM_Humana_ret_5d"], "is_new": true}, {"model_id": "new_h3_STRESS_XGBoost_N12_t2", "algo": "XGBoost", "regime": "STRESS", "horizon": 3, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "XLB_Materials_zscore_60d", "SBUX_zscore_60d", "SO_SouthernCo_ret_5d", "US30Y_Rate_ret_20d", "CPB_CampbellSoup_ret_5d", "NOC_Northrop_ret_20d", "EXC_Exelon_ret_1d", "VOD_Vodafone_zscore_60d", "LMT_LockheedMartin_vol_20d", "QQQ_vol_20d"], "is_new": true}, {"model_id": "new_h3_STRESS_XGBoost_N12_t3", "algo": "XGBoost", "regime": "STRESS", "horizon": 3, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "vix_mean_abs_ret_5d", "JNJ_ret_1d", "XOM_ret_1d", "VRP_ma5", "AMGN_Amgen_ret_1d", "AMD_ret_5d", "gjr_condvar_h1", "ES_Evergy_ret_1d", "SLB_Schlumberger_ret_5d", "HUM_Humana_ret_5d"], "is_new": true}, {"model_id": "new_h3_STRESS_XGBoost_N12_t4", "algo": "XGBoost", "regime": "STRESS", "horizon": 3, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "SJM_JM_Smucker_ret_1d", "AXP_Amex_ret_20d", "DAX_Germany_vol_20d", "TM_Telephone_vol_20d", "spx_abs_ret_max_5d", "EWM_Malaysia_ret_1d", "XOM_ret_20d", "CPB_CampbellSoup_ret_20d", "EWQ_France_zscore_60d", "SCHW_Schwab_ret_5d"], "is_new": true}, {"model_id": "new_h3_STRESS_XGBoost_N12_t5", "algo": "XGBoost", "regime": "STRESS", "horizon": 3, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "ASX_Australia_ret_5d", "EWM_Malaysia_ret_1d", "BLK_BlackRock_zscore_60d", "JNJ_ret_1d", "vix_acceleration_1d", "QQQ_vol_20d", "US3Y_Rate_ret_5d", "EOG_EOGResources_ret_5d", "IYR_US_REIT2_zscore_60d", "heston_var_ev_h3"], "is_new": true}, {"model_id": "new_h3_STRESS_XGBoost_N12_t6", "algo": "XGBoost", "regime": "STRESS", "horizon": 3, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWS_Singapore_ret_5d", "LLY_zscore_60d", "T10Y2Y_Spread_ret_5d", "Core_CPI_zscore_60d", "EWM_Malaysia_zscore_60d", "NEE_NextEra_ret_20d", "SJM_JM_Smucker_ret_5d", "EMR_Emerson_ret_20d", "AXP_Amex_vol_20d", "CCI_CrownCastle_vol_20d"], "is_new": true}, {"model_id": "new_h3_STRESS_XGBoost_N12_t7", "algo": "XGBoost", "regime": "STRESS", "horizon": 3, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "XLV_Health_zscore_60d", "NVDA_vol_20d", "US1Y_Rate_ret_20d", "MO_AltriaMG_ret_1d", "HD_ret_5d", "Michigan_Sentiment_ret_20d", "EOG_EOGResources_vol_20d", "CCI_CrownCastle_vol_20d", "CMCSA_ret_1d", "MSTR_Bitcoin3_ret_5d"], "is_new": true}, {"model_id": "new_h3_STRESS_XGBoost_N15_t0", "algo": "XGBoost", "regime": "STRESS", "horizon": 3, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "ENB_EnbridgeInc_ret_1d", "AXP_Amex_ret_20d", "SO_SouthernCo_ret_5d", "DHR_ret_1d", "CPB_CampbellSoup_zscore_60d", "DE_Deere_ret_5d", "GD_GeneralDynamics_zscore_60d", "AMD_ret_1d", "PFE_ret_1d", "spx_momentum_3d", "MSTR_Bitcoin3_ret_1d", "Nikkei_Japan_zscore_60d", "HD_ret_20d"], "is_new": true}, {"model_id": "new_h3_STRESS_XGBoost_N15_t1", "algo": "XGBoost", "regime": "STRESS", "horizon": 3, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EFFR_ret_1d", "US5Y_Rate_ret_5d", "DAX_Germany_zscore_60d", "Core_CPI_zscore_60d", "EWS_Singapore_ret_5d", "IYR_US_REIT2_zscore_60d", "AXP_Amex_ret_20d", "heston_ev_h3", "CPB_CampbellSoup_ret_5d", "CPB_CampbellSoup_vol_20d", "EWQ_France_zscore_60d", "GE_ret_1d", "EXC_Exelon_ret_1d"], "is_new": true}, {"model_id": "new_h3_STRESS_XGBoost_N15_t2", "algo": "XGBoost", "regime": "STRESS", "horizon": 3, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "VRP_ma5", "DE_Deere_ret_5d", "US3M_Rate_vol_20d", "CI_Cigna_vol_20d", "TED_Spread_vol_20d", "HD_ret_5d", "SLB_Schlumberger_ret_5d", "US7Y_Rate_ret_20d", "EXC_Exelon_ret_1d", "LOW_Lowes_ret_20d", "GE_ret_1d", "vix_mean_abs_ret_5d", "HD_ret_1d"], "is_new": true}, {"model_id": "new_h3_STRESS_XGBoost_N15_t3", "algo": "XGBoost", "regime": "STRESS", "horizon": 3, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "HUM_Humana_ret_5d", "PLD_Prologis_ret_5d", "DIS_vol_20d", "spx_momentum_3d", "BDX_Becton_Dickinson_ret_20d", "HD_ret_5d", "SBUX_zscore_60d", "EWY_Korea_zscore_60d", "HangSeng_HK_vol_20d", "DAX_Germany_zscore_60d", "CPB_CampbellSoup_zscore_60d", "PAYX_Paychex_vol_20d", "LOW_Lowes_ret_5d"], "is_new": true}, {"model_id": "new_h3_STRESS_XGBoost_N15_t4", "algo": "XGBoost", "regime": "STRESS", "horizon": 3, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "FedFunds_zscore_60d", "MSTR_Bitcoin3_ret_5d", "IWM_SmallCap_vol_20d", "CPB_CampbellSoup_ret_5d", "HD_ret_20d", "HangSeng_HK_ret_5d", "EXC_Exelon_zscore_60d", "HangSeng_HK_vol_20d", "NVDA_vol_20d", "LOW_Lowes_ret_5d", "SBUX_vol_20d", "IBEX_Spain_ret_20d", "SPY_zscore_60d"], "is_new": true}, {"model_id": "new_h3_STRESS_XGBoost_N15_t5", "algo": "XGBoost", "regime": "STRESS", "horizon": 3, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "INTC_ret_5d", "DE_Deere_vol_20d", "SLB_Schlumberger_ret_5d", "TGT_Target_zscore_60d", "vix_acceleration_1d", "MS_MorganStanley_ret_5d", "HangSeng_HK_ret_1d", "MSTR_Bitcoin3_ret_20d", "AMGN_Amgen_ret_1d", "US30Y_Rate_ret_20d", "XLK_Tech_zscore_60d", "EWM_Malaysia_vol_20d", "AMZN_ret_5d"], "is_new": true}, {"model_id": "new_h3_STRESS_XGBoost_N15_t6", "algo": "XGBoost", "regime": "STRESS", "horizon": 3, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "US1Y_Rate_ret_5d", "HUM_Humana_ret_5d", "EFFR_ret_1d", "Brent_Oil_FRED_ret_5d", "EWQ_France_ret_20d", "VVIX_ret_20d", "CPB_CampbellSoup_ret_5d", "EWM_Malaysia_ret_1d", "ORCL_vol_20d", "XLF_Fin_vol_20d", "XLY_Disc_vol_20d", "SJM_JM_Smucker_ret_1d", "JNJ_ret_1d"], "is_new": true}, {"model_id": "new_h3_STRESS_XGBoost_N15_t7", "algo": "XGBoost", "regime": "STRESS", "horizon": 3, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "SBUX_vol_20d", "SBUX_ret_5d", "DE_Deere_ret_5d", "ORCL_zscore_60d", "DAX_Germany_vol_20d", "LOW_Lowes_ret_5d", "EWY_Korea_zscore_60d", "DOW_Price_zscore_60d", "EMR_Emerson_ret_20d", "EXC_Exelon_ret_1d", "Core_CPI_zscore_60d", "ENB_EnbridgeInc_ret_1d", "INTC_ret_5d"], "is_new": true}, {"model_id": "new_h3_STRESS_XGBoost_N20_t0", "algo": "XGBoost", "regime": "STRESS", "horizon": 3, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "CPB_CampbellSoup_ret_5d", "ES_Evergy_ret_1d", "HD_ret_5d", "QQQ_vol_20d", "DHR_ret_1d", "EQR_Equity_ret_1d", "EWM_Malaysia_vol_20d", "PCAR_PaccarInc_ret_5d", "DIS_vol_20d", "AXP_Amex_ret_20d", "PAYX_Paychex_ret_20d", "HangSeng_HK_ret_1d", "BTI_BritishAmerican_ret_5d", "WTI_Oil_FRED_zscore_60d", "SBUX_ret_5d", "LLY_zscore_60d", "EWL_Switzerland_vol_20d", "Retail_Sales_zscore_60d"], "is_new": true}, {"model_id": "new_h3_STRESS_XGBoost_N20_t1", "algo": "XGBoost", "regime": "STRESS", "horizon": 3, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EMR_Emerson_ret_20d", "EWQ_France_ret_20d", "VVIX_ret_20d", "XLB_Materials_zscore_60d", "US1Y_Rate_ret_5d", "spx_vol_5d", "XOM_ret_20d", "SBUX_zscore_60d", "PCAR_PaccarInc_ret_5d", "EWH_HongKong_ret_5d", "EWM_Malaysia_ret_1d", "MRK_Merck_zscore_60d", "EFFR_vol_20d", "CPB_CampbellSoup_ret_20d", "3M_ret_5d", "TED_Spread_zscore_60d", "JNJ_ret_1d", "EWA_Australia_ret_1d"], "is_new": true}, {"model_id": "new_h3_STRESS_XGBoost_N20_t2", "algo": "XGBoost", "regime": "STRESS", "horizon": 3, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "TXN_vol_20d", "EQIX_Equinix_ret_5d", "NOC_Northrop_ret_20d", "EWM_Malaysia_ret_1d", "EFFR_vol_20d", "CLX_Clorox_vol_20d", "TED_Spread_zscore_60d", "XLB_Materials_zscore_60d", "HD_ret_1d", "MO_AltriaMG_ret_1d", "CPB_CampbellSoup_ret_5d", "BLK_BlackRock_zscore_60d", "US1Y_Rate_ret_5d", "T10Y2Y_Spread_ret_5d", "LLY_zscore_60d", "SLB_Schlumberger_ret_1d", "US3M_Rate_vol_20d", "US6M_Rate_ret_20d"], "is_new": true}, {"model_id": "new_h3_STRESS_XGBoost_N20_t3", "algo": "XGBoost", "regime": "STRESS", "horizon": 3, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "XLB_Materials_zscore_60d", "XLY_Disc_vol_20d", "GE_ret_1d", "EQR_Equity_ret_1d", "US30Y_Rate_ret_20d", "Core_PCE_zscore_60d", "SJM_JM_Smucker_ret_1d", "EWQ_France_ret_20d", "HD_ret_5d", "DE_Deere_vol_20d", "heston_var_ev_h5", "3M_ret_5d", "NFCI_ret_5d", "EXC_Exelon_ret_1d", "EWA_Australia_ret_1d", "EWG_Germany_ret_20d", "spx_momentum_3d", "VOD_Vodafone_zscore_60d"], "is_new": true}, {"model_id": "new_h3_STRESS_XGBoost_N20_t4", "algo": "XGBoost", "regime": "STRESS", "horizon": 3, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "IBEX_Spain_ret_20d", "EWC_Canada_zscore_60d", "EWA_Australia_ret_1d", "HD_ret_20d", "US3M_Rate_vol_20d", "PCAR_PaccarInc_ret_5d", "CPB_CampbellSoup_ret_5d", "VVIX_ret_20d", "PAYX_Paychex_ret_20d", "DAX_Germany_zscore_60d", "XOM_ret_20d", "AMD_ret_5d", "GE_ret_1d", "AXP_Amex_vol_20d", "EWL_Switzerland_vol_20d", "ES_Evergy_ret_1d", "HD_zscore_60d", "DIS_vol_20d"], "is_new": true}, {"model_id": "new_h3_STRESS_XGBoost_N20_t5", "algo": "XGBoost", "regime": "STRESS", "horizon": 3, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "XLK_Tech_zscore_60d", "heston_var_ev_h7", "CCI_CrownCastle_vol_20d", "PFE_ret_1d", "EWM_Malaysia_ret_1d", "CTAS_Cintas_vol_20d", "MSTR_Bitcoin3_ret_1d", "EWA_Australia_zscore_60d", "EWL_Switzerland_zscore_60d", "DE_Deere_vol_20d", "HUM_Humana_ret_5d", "EWM_Malaysia_vol_20d", "3M_ret_5d", "spx_momentum_3d", "SPY_zscore_60d", "spx_abs_ret_max_5d", "vix_acceleration_1d", "SJM_JM_Smucker_ret_1d"], "is_new": true}, {"model_id": "new_h3_STRESS_XGBoost_N20_t6", "algo": "XGBoost", "regime": "STRESS", "horizon": 3, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "SJM_JM_Smucker_ret_5d", "EWQ_France_zscore_60d", "US3Y_Rate_ret_5d", "AORD_AUS_zscore_60d", "SBUX_vol_20d", "Core_PCE_zscore_60d", "DE_Deere_vol_20d", "CI_Cigna_vol_20d", "ENB_EnbridgeInc_ret_1d", "LUV_SouthwestAir_ret_5d", "SCHW_Schwab_ret_5d", "XLY_Disc_vol_20d", "TM_Telephone_vol_20d", "VOD_Vodafone_zscore_60d", "Nikkei_Japan_zscore_60d", "WTI_Oil_FRED_zscore_60d", "Brent_Oil_FRED_ret_5d", "AXP_Amex_vol_20d"], "is_new": true}, {"model_id": "new_h3_STRESS_XGBoost_N20_t7", "algo": "XGBoost", "regime": "STRESS", "horizon": 3, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "PPL_PPL_ret_1d", "EMR_Emerson_ret_20d", "HD_ret_1d", "HD_zscore_60d", "hmm_p_stress", "XOM_ret_1d", "LOW_Lowes_ret_5d", "EQIX_Equinix_ret_5d", "ASX_Australia_vol_20d", "ORCL_zscore_60d", "MSTR_Bitcoin3_ret_5d", "US3Y_Rate_ret_5d", "US7Y_Rate_ret_20d", "XOM_ret_20d", "Brent_Oil_FRED_ret_5d", "LOW_Lowes_ret_20d", "DHR_vol_20d", "TM_Telephone_vol_20d"], "is_new": true}, {"model_id": "new_h3_STRESS_XGBoost_N25_t0", "algo": "XGBoost", "regime": "STRESS", "horizon": 3, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "SCHW_Schwab_ret_5d", "SLB_Schlumberger_ret_1d", "US5Y_Rate_ret_5d", "LOW_Lowes_ret_20d", "CPB_CampbellSoup_zscore_60d", "Nikkei_Japan_vol_20d", "Michigan_Sentiment_ret_20d", "spx_momentum_3d", "BDX_Becton_Dickinson_ret_20d", "spx_abs_ret_max_5d", "TGT_Target_zscore_60d", "AXP_Amex_vol_20d", "BA_ret_1d", "MRK_Merck_zscore_60d", "GILD_Gilead_ret_20d", "HUM_Humana_ret_5d", "AMT_AmericanTower_ret_1d", "DHR_vol_20d", "PLD_Prologis_ret_5d", "ORCL_zscore_60d", "MS_MorganStanley_ret_1d", "BLK_BlackRock_zscore_60d", "Core_CPI_zscore_60d"], "is_new": true}, {"model_id": "new_h3_STRESS_XGBoost_N25_t1", "algo": "XGBoost", "regime": "STRESS", "horizon": 3, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "MS_MorganStanley_ret_1d", "DHR_ret_1d", "Industrial_Production_zscore_60d", "PPL_PPL_ret_1d", "MO_AltriaMG_ret_1d", "CMCSA_ret_1d", "AMZN_ret_5d", "ASX_Australia_ret_5d", "AMGN_Amgen_ret_1d", "EWL_Switzerland_vol_20d", "MSTR_Bitcoin3_ret_5d", "NFCI_ret_5d", "TM_Telephone_ret_1d", "LMT_LockheedMartin_ret_1d", "spx_momentum_3d", "IBEX_Spain_ret_20d", "EWQ_France_zscore_60d", "US7Y_Rate_ret_20d", "QQQ_vol_20d", "EWQ_France_ret_20d", "T_ret_1d", "GE_ret_1d", "PAYX_Paychex_zscore_60d"], "is_new": true}, {"model_id": "new_h3_STRESS_XGBoost_N25_t2", "algo": "XGBoost", "regime": "STRESS", "horizon": 3, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "SJM_JM_Smucker_ret_5d", "Brent_Oil_FRED_ret_5d", "MS_MorganStanley_ret_1d", "EWM_Malaysia_vol_20d", "AXP_Amex_vol_20d", "EQIX_Equinix_ret_5d", "CPB_CampbellSoup_zscore_60d", "EOG_EOGResources_vol_20d", "VRP_ma5", "vix_acceleration_1d", "TM_Telephone_ret_1d", "EFFR_vol_20d", "EWA_Australia_zscore_60d", "NEE_NextEra_ret_20d", "XOM_ret_1d", "QQQ_vol_20d", "DE_Deere_vol_20d", "IYM_BasicMaterials_ret_20d", "CTAS_Cintas_vol_20d", "CMCSA_ret_1d", "DHR_ret_1d", "T10Y2Y_Spread_ret_5d", "ASX_Australia_ret_5d"], "is_new": true}, {"model_id": "new_h3_STRESS_XGBoost_N25_t3", "algo": "XGBoost", "regime": "STRESS", "horizon": 3, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EFFR_ret_1d", "MSTR_Bitcoin3_ret_20d", "vix_mean_abs_ret_5d", "EWM_Malaysia_zscore_60d", "ORCL_zscore_60d", "AMZN_ret_5d", "LOW_Lowes_ret_5d", "ASX_Australia_vol_20d", "MSTR_Bitcoin3_ret_5d", "PCAR_PaccarInc_ret_5d", "HUM_Humana_ret_5d", "US1Y_Rate_ret_5d", "EOG_EOGResources_vol_20d", "LMT_LockheedMartin_ret_1d", "EFFR_vol_20d", "spx_vol_5d", "US5Y_Rate_ret_5d", "XLV_Health_zscore_60d", "IWM_SmallCap_vol_20d", "HD_ret_20d", "MS_MorganStanley_ret_5d", "Industrial_Production_zscore_60d", "HD_ret_1d"], "is_new": true}, {"model_id": "new_h3_STRESS_XGBoost_N25_t4", "algo": "XGBoost", "regime": "STRESS", "horizon": 3, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "Nikkei_Japan_zscore_60d", "XLB_Materials_zscore_60d", "CMCSA_ret_1d", "HangSeng_HK_ret_1d", "MO_AltriaMG_ret_1d", "LMT_LockheedMartin_vol_20d", "VVIX_ret_20d", "BA_ret_1d", "EWG_Germany_vol_20d", "MSTR_Bitcoin3_ret_5d", "Industrial_Production_zscore_60d", "EWM_Malaysia_ret_1d", "hmm_p_stress", "HD_ret_20d", "EFFR_vol_20d", "heston_var_ev_h3", "ORCL_zscore_60d", "XLV_Health_zscore_60d", "EXC_Exelon_ret_1d", "ORCL_vol_20d", "NWL_Newell_ret_20d", "PAYX_Paychex_zscore_60d", "heston_ev_h3"], "is_new": true}, {"model_id": "new_h3_STRESS_XGBoost_N25_t5", "algo": "XGBoost", "regime": "STRESS", "horizon": 3, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "SBUX_ret_5d", "MS_MorganStanley_ret_1d", "Brent_Oil_FRED_ret_20d", "SCHW_Schwab_ret_5d", "HangSeng_HK_ret_5d", "heston_var_ev_h5", "ORCL_zscore_60d", "VOD_Vodafone_zscore_60d", "AMD_ret_5d", "HD_ret_1d", "LOW_Lowes_ret_20d", "IYR_US_REIT2_zscore_60d", "ENB_EnbridgeInc_ret_1d", "DHR_vol_20d", "HUM_Humana_ret_5d", "EOG_EOGResources_ret_5d", "AXP_Amex_ret_20d", "LOW_Lowes_ret_5d", "VVIX_ret_20d", "Nikkei_Japan_vol_20d", "EXC_Exelon_zscore_60d", "DIS_vol_20d", "LUV_SouthwestAir_ret_5d"], "is_new": true}, {"model_id": "new_h3_STRESS_XGBoost_N25_t6", "algo": "XGBoost", "regime": "STRESS", "horizon": 3, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "GE_ret_1d", "EWM_Malaysia_vol_20d", "SBUX_ret_5d", "CI_Cigna_vol_20d", "XLY_Disc_vol_20d", "MS_MorganStanley_zscore_60d", "AVB_AvalonBay_zscore_60d", "EWA_Australia_ret_1d", "HD_zscore_60d", "SLB_Schlumberger_ret_5d", "XOM_ret_1d", "SBUX_vol_20d", "EWL_Switzerland_zscore_60d", "BTI_BritishAmerican_ret_5d", "CPB_CampbellSoup_zscore_60d", "Nikkei_Japan_vol_20d", "AXP_Amex_ret_20d", "DAX_Germany_vol_20d", "LMT_LockheedMartin_ret_1d", "MSTR_Bitcoin3_ret_20d", "JNJ_ret_1d", "EWQ_France_ret_20d", "US3M_Rate_vol_20d"], "is_new": true}, {"model_id": "new_h3_STRESS_XGBoost_N25_t7", "algo": "XGBoost", "regime": "STRESS", "horizon": 3, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "CPB_CampbellSoup_ret_20d", "heston_var_ev_h3", "INTC_ret_5d", "HD_ret_1d", "EQR_Equity_ret_1d", "EXC_Exelon_ret_1d", "3M_vol_20d", "CMCSA_ret_1d", "LLY_zscore_60d", "AMT_AmericanTower_ret_1d", "EXC_Exelon_zscore_60d", "EWJ_Japan_vol_20d", "PLD_Prologis_ret_5d", "XLK_Tech_zscore_60d", "BLK_BlackRock_zscore_60d", "TED_Spread_vol_20d", "NOC_Northrop_ret_20d", "T_ret_1d", "AMZN_ret_5d", "TED_Spread_zscore_60d", "SO_SouthernCo_ret_5d", "EWQ_France_zscore_60d", "Industrial_Production_zscore_60d"], "is_new": true}, {"model_id": "new_h3_STRESS_XGBoost_N30_t0", "algo": "XGBoost", "regime": "STRESS", "horizon": 3, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "Nikkei_Japan_vol_20d", "EXC_Exelon_zscore_60d", "ITT_ITTInc_ret_5d", "FedFunds_zscore_60d", "heston_ev_h3", "US30Y_Rate_ret_20d", "PAYX_Paychex_vol_20d", "MRK_Merck_zscore_60d", "XLK_Tech_zscore_60d", "heston_var_ev_h7", "US5Y_Rate_ret_5d", "US3M_Rate_vol_20d", "SBUX_ret_5d", "vix_acceleration_1d", "DE_Deere_vol_20d", "LMT_LockheedMartin_ret_1d", "HD_zscore_60d", "EMR_Emerson_ret_20d", "US6M_Rate_ret_20d", "BDX_Becton_Dickinson_ret_20d", "HangSeng_HK_ret_5d", "EWJ_Japan_vol_20d", "MSTR_Bitcoin3_ret_5d", "TED_Spread_zscore_60d", "ENB_EnbridgeInc_ret_1d", "Retail_Sales_zscore_60d", "IWM_SmallCap_vol_20d", "Michigan_Sentiment_ret_20d"], "is_new": true}, {"model_id": "new_h3_STRESS_XGBoost_N30_t1", "algo": "XGBoost", "regime": "STRESS", "horizon": 3, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "BLK_BlackRock_zscore_60d", "PAYX_Paychex_zscore_60d", "T10Y2Y_Spread_ret_5d", "heston_var_ev_h5", "Nikkei_Japan_zscore_60d", "SBUX_ret_5d", "EQR_Equity_ret_1d", "INTC_ret_5d", "EWM_Malaysia_zscore_60d", "BDX_Becton_Dickinson_ret_20d", "SLB_Schlumberger_ret_5d", "CPB_CampbellSoup_zscore_60d", "IYR_US_REIT2_zscore_60d", "TED_Spread_zscore_60d", "3M_vol_20d", "TXN_vol_20d", "Michigan_Sentiment_ret_20d", "ORCL_zscore_60d", "spx_vol_5d", "VVIX_ret_20d", "EXC_Exelon_zscore_60d", "MRK_Merck_zscore_60d", "XOM_ret_20d", "US7Y_Rate_ret_20d", "EWL_Switzerland_zscore_60d", "EFFR_vol_20d", "HD_ret_20d", "NFCI_ret_5d"], "is_new": true}, {"model_id": "new_h3_STRESS_XGBoost_N30_t2", "algo": "XGBoost", "regime": "STRESS", "horizon": 3, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "Industrial_Production_zscore_60d", "EOG_EOGResources_ret_5d", "EWL_Switzerland_vol_20d", "ORCL_zscore_60d", "CPB_CampbellSoup_zscore_60d", "AMT_AmericanTower_ret_1d", "AMD_ret_1d", "DIS_vol_20d", "XOM_ret_1d", "MO_AltriaMG_ret_1d", "XLK_Tech_zscore_60d", "MSTR_Bitcoin3_ret_5d", "EWA_Australia_ret_1d", "INTC_ret_5d", "Brent_Oil_FRED_ret_20d", "US30Y_Rate_ret_20d", "SBUX_vol_20d", "LMT_LockheedMartin_ret_1d", "PAYX_Paychex_zscore_60d", "BDX_Becton_Dickinson_ret_20d", "LUV_SouthwestAir_ret_5d", "EWG_Germany_vol_20d", "EQR_Equity_ret_1d", "AVB_AvalonBay_zscore_60d", "US3M_Rate_vol_20d", "EWS_Singapore_ret_5d", "MSTR_Bitcoin3_ret_1d", "Brent_Oil_FRED_ret_5d"], "is_new": true}, {"model_id": "new_h3_STRESS_XGBoost_N30_t3", "algo": "XGBoost", "regime": "STRESS", "horizon": 3, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "IBEX_Spain_ret_20d", "NEE_NextEra_ret_20d", "GILD_Gilead_ret_20d", "CPB_CampbellSoup_zscore_60d", "XOM_ret_1d", "SBUX_ret_5d", "EFFR_vol_20d", "US3M_Rate_vol_20d", "HD_ret_5d", "Brent_Oil_FRED_ret_20d", "BDX_Becton_Dickinson_ret_20d", "EWJ_Japan_vol_20d", "Nikkei_Japan_vol_20d", "SCHW_Schwab_ret_5d", "IWM_SmallCap_vol_20d", "TED_Spread_zscore_60d", "AMD_ret_5d", "MS_MorganStanley_ret_1d", "US3M_Rate_zscore_60d", "EWL_Switzerland_zscore_60d", "EFFR_ret_1d", "TED_Spread_vol_20d", "GD_GeneralDynamics_zscore_60d", "AVB_AvalonBay_zscore_60d", "heston_ev_h3", "CMCSA_ret_1d", "PCAR_PaccarInc_ret_5d", "Brent_Oil_FRED_ret_5d"], "is_new": true}, {"model_id": "new_h3_STRESS_XGBoost_N30_t4", "algo": "XGBoost", "regime": "STRESS", "horizon": 3, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "AORD_AUS_zscore_60d", "XLY_Disc_vol_20d", "PFE_ret_1d", "CPB_CampbellSoup_ret_5d", "TED_Spread_zscore_60d", "FedFunds_zscore_60d", "NFCI_ret_5d", "spx_vol_5d", "VRP_ma5", "BDX_Becton_Dickinson_ret_20d", "US1Y_Rate_ret_20d", "PLD_Prologis_ret_5d", "QQQ_vol_20d", "EWM_Malaysia_zscore_60d", "XOM_ret_1d", "EWL_Switzerland_vol_20d", "CPB_CampbellSoup_zscore_60d", "NOC_Northrop_ret_20d", "US3Y_Rate_ret_5d", "DHR_vol_20d", "PAYX_Paychex_vol_20d", "LLY_zscore_60d", "EOG_EOGResources_ret_5d", "SBUX_zscore_60d", "IBEX_Spain_ret_20d", "heston_var_ev_h7", "AXP_Amex_vol_20d", "CI_Cigna_vol_20d"], "is_new": true}, {"model_id": "new_h3_STRESS_XGBoost_N30_t5", "algo": "XGBoost", "regime": "STRESS", "horizon": 3, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "PPL_PPL_ret_1d", "HD_zscore_60d", "BA_ret_1d", "3M_ret_5d", "VVIX_ret_20d", "PG_ret_20d", "BDX_Becton_Dickinson_ret_20d", "BTI_BritishAmerican_ret_20d", "SO_SouthernCo_ret_5d", "EWL_Switzerland_vol_20d", "DE_Deere_ret_5d", "MO_AltriaMG_ret_1d", "MS_MorganStanley_zscore_60d", "MSTR_Bitcoin3_ret_20d", "Brent_Oil_FRED_ret_20d", "TXN_vol_20d", "AMD_ret_1d", "MSTR_Bitcoin3_ret_1d", "CPB_CampbellSoup_ret_20d", "spx_momentum_3d", "TED_Spread_vol_20d", "Industrial_Production_zscore_60d", "EWS_Singapore_ret_5d", "PCAR_PaccarInc_ret_5d", "AVB_AvalonBay_zscore_60d", "3M_vol_20d", "SJM_JM_Smucker_ret_1d", "BLK_BlackRock_zscore_60d"], "is_new": true}, {"model_id": "new_h3_STRESS_XGBoost_N30_t6", "algo": "XGBoost", "regime": "STRESS", "horizon": 3, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "DHR_ret_1d", "DE_Deere_ret_5d", "Core_CPI_zscore_60d", "EOG_EOGResources_ret_5d", "MS_MorganStanley_zscore_60d", "US30Y_Rate_ret_20d", "EWL_Switzerland_vol_20d", "XLV_Health_zscore_60d", "AVB_AvalonBay_zscore_60d", "INTC_ret_5d", "CPB_CampbellSoup_ret_20d", "ASX_Australia_vol_20d", "DAX_Germany_vol_20d", "Brent_Oil_FRED_ret_5d", "TXN_vol_20d", "BTI_BritishAmerican_ret_5d", "MSTR_Bitcoin3_ret_1d", "US3M_Rate_vol_20d", "EWC_Canada_zscore_60d", "GD_GeneralDynamics_zscore_60d", "VOD_Vodafone_zscore_60d", "CTAS_Cintas_vol_20d", "3M_vol_20d", "MSTR_Bitcoin3_ret_20d", "PG_ret_20d", "ORCL_vol_20d", "SPY_zscore_60d", "Retail_Sales_zscore_60d"], "is_new": true}, {"model_id": "new_h3_STRESS_XGBoost_N30_t7", "algo": "XGBoost", "regime": "STRESS", "horizon": 3, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "XLV_Health_zscore_60d", "HangSeng_HK_vol_20d", "CCI_CrownCastle_vol_20d", "spx_vol_5d", "NWL_Newell_ret_20d", "EWM_Malaysia_vol_20d", "EQIX_Equinix_ret_5d", "US30Y_Rate_ret_20d", "DOW_Price_zscore_60d", "heston_var_ev_h3", "LOW_Lowes_ret_5d", "CTAS_Cintas_vol_20d", "US3M_Rate_zscore_60d", "MSTR_Bitcoin3_ret_20d", "EXC_Exelon_ret_1d", "DE_Deere_vol_20d", "VRP_ma5", "IYM_BasicMaterials_ret_20d", "SLB_Schlumberger_ret_1d", "MSTR_Bitcoin3_ret_5d", "DAX_Germany_vol_20d", "Nikkei_Japan_zscore_60d", "PAYX_Paychex_vol_20d", "DIS_vol_20d", "CPB_CampbellSoup_zscore_60d", "BTI_BritishAmerican_ret_20d", "SBUX_vol_20d", "AMT_AmericanTower_ret_1d"], "is_new": true}, {"model_id": "new_h3_STRESS_LightGBM_N5_t0", "algo": "LightGBM", "regime": "STRESS", "horizon": 3, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "VVIX_ret_20d", "MS_MorganStanley_ret_1d", "INTC_ret_1d"], "is_new": true}, {"model_id": "new_h3_STRESS_LightGBM_N5_t1", "algo": "LightGBM", "regime": "STRESS", "horizon": 3, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EFFR_vol_20d", "CI_Cigna_vol_20d", "SPY_zscore_60d"], "is_new": true}, {"model_id": "new_h3_STRESS_LightGBM_N5_t2", "algo": "LightGBM", "regime": "STRESS", "horizon": 3, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "heston_ev_h3", "TED_Spread_vol_20d", "AXP_Amex_vol_20d"], "is_new": true}, {"model_id": "new_h3_STRESS_LightGBM_N5_t3", "algo": "LightGBM", "regime": "STRESS", "horizon": 3, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "3M_ret_5d", "EOG_EOGResources_ret_5d", "HUM_Humana_ret_5d"], "is_new": true}, {"model_id": "new_h3_STRESS_LightGBM_N5_t4", "algo": "LightGBM", "regime": "STRESS", "horizon": 3, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWL_Switzerland_vol_20d", "EWS_Singapore_ret_5d", "MRK_Merck_zscore_60d"], "is_new": true}, {"model_id": "new_h3_STRESS_LightGBM_N5_t5", "algo": "LightGBM", "regime": "STRESS", "horizon": 3, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "gjr_condvar_h1", "ITT_ITTInc_ret_5d", "HD_ret_5d"], "is_new": true}, {"model_id": "new_h3_STRESS_LightGBM_N5_t6", "algo": "LightGBM", "regime": "STRESS", "horizon": 3, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWA_Australia_zscore_60d", "EFFR_ret_1d", "3M_ret_5d"], "is_new": true}, {"model_id": "new_h3_STRESS_LightGBM_N5_t7", "algo": "LightGBM", "regime": "STRESS", "horizon": 3, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWG_Germany_vol_20d", "XLV_Health_zscore_60d", "heston_var_ev_h3"], "is_new": true}, {"model_id": "new_h3_STRESS_LightGBM_N8_t0", "algo": "LightGBM", "regime": "STRESS", "horizon": 3, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "TED_Spread_vol_20d", "NOC_Northrop_ret_20d", "PAYX_Paychex_zscore_60d", "NWL_Newell_ret_20d", "PG_ret_20d", "HD_ret_1d"], "is_new": true}, {"model_id": "new_h3_STRESS_LightGBM_N8_t1", "algo": "LightGBM", "regime": "STRESS", "horizon": 3, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "US1Y_Rate_ret_20d", "AMD_ret_1d", "SJM_JM_Smucker_ret_1d", "EWA_Australia_zscore_60d", "XLV_Health_zscore_60d", "BLK_BlackRock_zscore_60d"], "is_new": true}, {"model_id": "new_h3_STRESS_LightGBM_N8_t2", "algo": "LightGBM", "regime": "STRESS", "horizon": 3, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "ASX_Australia_ret_5d", "US6M_Rate_ret_20d", "MSTR_Bitcoin3_ret_5d", "NOC_Northrop_ret_20d", "MS_MorganStanley_ret_5d", "PAYX_Paychex_vol_20d"], "is_new": true}, {"model_id": "new_h3_STRESS_LightGBM_N8_t3", "algo": "LightGBM", "regime": "STRESS", "horizon": 3, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "gjr_condvar_h1", "EFFR_vol_20d", "XLF_Fin_vol_20d", "ORCL_vol_20d", "hmm_p_stress", "IBEX_Spain_ret_20d"], "is_new": true}, {"model_id": "new_h3_STRESS_LightGBM_N8_t4", "algo": "LightGBM", "regime": "STRESS", "horizon": 3, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "SCHW_Schwab_ret_5d", "DAX_Germany_vol_20d", "NOC_Northrop_ret_20d", "HangSeng_HK_ret_1d", "VVIX_ret_20d", "IWM_SmallCap_vol_20d"], "is_new": true}, {"model_id": "new_h3_STRESS_LightGBM_N8_t5", "algo": "LightGBM", "regime": "STRESS", "horizon": 3, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "NEE_NextEra_ret_20d", "LLY_zscore_60d", "PLD_Prologis_ret_5d", "PG_ret_20d", "Core_PCE_zscore_60d", "EWS_Singapore_ret_5d"], "is_new": true}, {"model_id": "new_h3_STRESS_LightGBM_N8_t6", "algo": "LightGBM", "regime": "STRESS", "horizon": 3, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWC_Canada_zscore_60d", "EXC_Exelon_ret_1d", "ENB_EnbridgeInc_ret_1d", "PLD_Prologis_ret_5d", "SJM_JM_Smucker_ret_5d", "US6M_Rate_ret_20d"], "is_new": true}, {"model_id": "new_h3_STRESS_LightGBM_N8_t7", "algo": "LightGBM", "regime": "STRESS", "horizon": 3, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "DIS_vol_20d", "EOG_EOGResources_vol_20d", "PLD_Prologis_ret_5d", "MSTR_Bitcoin3_ret_1d", "Nikkei_Japan_vol_20d", "ITT_ITTInc_ret_5d"], "is_new": true}, {"model_id": "new_h3_STRESS_LightGBM_N10_t0", "algo": "LightGBM", "regime": "STRESS", "horizon": 3, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "HangSeng_HK_vol_20d", "VVIX_ret_20d", "HD_zscore_60d", "Nikkei_Japan_vol_20d", "JNJ_ret_1d", "XOM_ret_20d", "GILD_Gilead_ret_20d", "heston_var_ev_h7"], "is_new": true}, {"model_id": "new_h3_STRESS_LightGBM_N10_t1", "algo": "LightGBM", "regime": "STRESS", "horizon": 3, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EFFR_ret_1d", "heston_var_ev_h5", "CPB_CampbellSoup_ret_5d", "HangSeng_HK_ret_1d", "EWM_Malaysia_zscore_60d", "MSTR_Bitcoin3_ret_5d", "NOC_Northrop_ret_20d", "IWM_SmallCap_vol_20d"], "is_new": true}, {"model_id": "new_h3_STRESS_LightGBM_N10_t2", "algo": "LightGBM", "regime": "STRESS", "horizon": 3, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "US1Y_Rate_ret_20d", "EOG_EOGResources_vol_20d", "EWL_Switzerland_vol_20d", "DE_Deere_vol_20d", "EWY_Korea_ret_20d", "HangSeng_HK_vol_20d", "IYR_US_REIT2_zscore_60d", "EWS_Singapore_ret_5d"], "is_new": true}, {"model_id": "new_h3_STRESS_LightGBM_N10_t3", "algo": "LightGBM", "regime": "STRESS", "horizon": 3, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "SO_SouthernCo_ret_5d", "EMR_Emerson_ret_20d", "DE_Deere_ret_5d", "TED_Spread_vol_20d", "WTI_Oil_FRED_zscore_60d", "NOC_Northrop_ret_20d", "LUV_SouthwestAir_ret_5d", "SBUX_ret_5d"], "is_new": true}, {"model_id": "new_h3_STRESS_LightGBM_N10_t4", "algo": "LightGBM", "regime": "STRESS", "horizon": 3, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "vix_acceleration_1d", "3M_vol_20d", "EWL_Switzerland_zscore_60d", "XLV_Health_zscore_60d", "T_ret_1d", "3M_ret_5d", "CTAS_Cintas_vol_20d", "EWG_Germany_ret_20d"], "is_new": true}, {"model_id": "new_h3_STRESS_LightGBM_N10_t5", "algo": "LightGBM", "regime": "STRESS", "horizon": 3, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "XLK_Tech_zscore_60d", "Brent_Oil_FRED_ret_5d", "LUV_SouthwestAir_ret_5d", "PCAR_PaccarInc_ret_5d", "SBUX_ret_5d", "AMD_ret_1d", "CPB_CampbellSoup_ret_5d", "IBEX_Spain_ret_20d"], "is_new": true}, {"model_id": "new_h3_STRESS_LightGBM_N10_t6", "algo": "LightGBM", "regime": "STRESS", "horizon": 3, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "TED_Spread_zscore_60d", "CPB_CampbellSoup_zscore_60d", "heston_ev_h3", "MSTR_Bitcoin3_ret_20d", "EFFR_ret_1d", "EWJ_Japan_vol_20d", "BTI_BritishAmerican_ret_5d", "LMT_LockheedMartin_vol_20d"], "is_new": true}, {"model_id": "new_h3_STRESS_LightGBM_N10_t7", "algo": "LightGBM", "regime": "STRESS", "horizon": 3, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "CTAS_Cintas_vol_20d", "LOW_Lowes_ret_20d", "TXN_vol_20d", "PG_ret_20d", "Michigan_Sentiment_ret_20d", "Industrial_Production_zscore_60d", "ASX_Australia_vol_20d", "US5Y_Rate_ret_5d"], "is_new": true}, {"model_id": "new_h3_STRESS_LightGBM_N12_t0", "algo": "LightGBM", "regime": "STRESS", "horizon": 3, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "heston_var_ev_h3", "LUV_SouthwestAir_ret_5d", "AMT_AmericanTower_ret_1d", "MO_AltriaMG_ret_1d", "ORCL_zscore_60d", "XLF_Fin_vol_20d", "DHR_ret_1d", "Core_PCE_zscore_60d", "BTI_BritishAmerican_ret_20d", "ASX_Australia_vol_20d"], "is_new": true}, {"model_id": "new_h3_STRESS_LightGBM_N12_t1", "algo": "LightGBM", "regime": "STRESS", "horizon": 3, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "heston_var_ev_h5", "US6M_Rate_ret_20d", "FedFunds_zscore_60d", "HUM_Humana_ret_5d", "EWL_Switzerland_zscore_60d", "AMD_ret_5d", "ASX_Australia_vol_20d", "DHR_ret_1d", "PFE_ret_1d", "SBUX_vol_20d"], "is_new": true}, {"model_id": "new_h3_STRESS_LightGBM_N12_t2", "algo": "LightGBM", "regime": "STRESS", "horizon": 3, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "IYR_US_REIT2_zscore_60d", "MS_MorganStanley_zscore_60d", "US3M_Rate_vol_20d", "HD_zscore_60d", "DIS_vol_20d", "PAYX_Paychex_vol_20d", "heston_var_ev_h5", "NFCI_ret_5d", "XOM_ret_20d", "EWA_Australia_ret_1d"], "is_new": true}, {"model_id": "new_h3_STRESS_LightGBM_N12_t3", "algo": "LightGBM", "regime": "STRESS", "horizon": 3, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWC_Canada_zscore_60d", "ENB_EnbridgeInc_ret_1d", "MS_MorganStanley_ret_5d", "PAYX_Paychex_ret_20d", "NOC_Northrop_ret_20d", "EWM_Malaysia_ret_1d", "heston_var_ev_h5", "CPB_CampbellSoup_ret_5d", "CLX_Clorox_vol_20d", "IBEX_Spain_ret_20d"], "is_new": true}, {"model_id": "new_h3_STRESS_LightGBM_N12_t4", "algo": "LightGBM", "regime": "STRESS", "horizon": 3, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "Industrial_Production_zscore_60d", "EWM_Malaysia_vol_20d", "VVIX_ret_20d", "SJM_JM_Smucker_ret_5d", "QQQ_vol_20d", "CCI_CrownCastle_vol_20d", "GILD_Gilead_ret_20d", "CPB_CampbellSoup_zscore_60d", "GD_GeneralDynamics_zscore_60d", "ES_Evergy_ret_1d"], "is_new": true}, {"model_id": "new_h3_STRESS_LightGBM_N12_t5", "algo": "LightGBM", "regime": "STRESS", "horizon": 3, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "AORD_AUS_zscore_60d", "MSTR_Bitcoin3_ret_1d", "vix_mean_abs_ret_5d", "EXC_Exelon_ret_1d", "MSTR_Bitcoin3_ret_20d", "CTAS_Cintas_vol_20d", "HD_ret_20d", "EWC_Canada_zscore_60d", "SLB_Schlumberger_ret_1d", "EWG_Germany_ret_20d"], "is_new": true}, {"model_id": "new_h3_STRESS_LightGBM_N12_t6", "algo": "LightGBM", "regime": "STRESS", "horizon": 3, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "HangSeng_HK_ret_1d", "DE_Deere_vol_20d", "TM_Telephone_vol_20d", "VVIX_ret_20d", "INTC_ret_1d", "IBEX_Spain_ret_20d", "BA_ret_1d", "LOW_Lowes_ret_20d", "TGT_Target_zscore_60d", "HD_ret_20d"], "is_new": true}, {"model_id": "new_h3_STRESS_LightGBM_N12_t7", "algo": "LightGBM", "regime": "STRESS", "horizon": 3, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "US1Y_Rate_ret_5d", "US7Y_Rate_ret_20d", "HD_zscore_60d", "SBUX_zscore_60d", "Industrial_Production_zscore_60d", "DE_Deere_vol_20d", "SLB_Schlumberger_ret_5d", "VOD_Vodafone_zscore_60d", "SBUX_ret_5d", "EMR_Emerson_ret_20d"], "is_new": true}, {"model_id": "new_h3_STRESS_LightGBM_N15_t0", "algo": "LightGBM", "regime": "STRESS", "horizon": 3, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "US3M_Rate_vol_20d", "US1Y_Rate_ret_5d", "VOD_Vodafone_zscore_60d", "TXN_vol_20d", "EQIX_Equinix_ret_5d", "ES_Evergy_ret_1d", "HD_ret_5d", "JNJ_ret_1d", "LLY_zscore_60d", "HUM_Humana_ret_5d", "AMZN_ret_5d", "hmm_p_stress", "HD_ret_1d"], "is_new": true}, {"model_id": "new_h3_STRESS_LightGBM_N15_t1", "algo": "LightGBM", "regime": "STRESS", "horizon": 3, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "CTAS_Cintas_vol_20d", "HangSeng_HK_ret_1d", "heston_ev_h3", "VRP_ma5", "XLK_Tech_zscore_60d", "M_Macys_vol_20d", "Core_PCE_zscore_60d", "QQQ_vol_20d", "PAYX_Paychex_vol_20d", "TED_Spread_vol_20d", "ITT_ITTInc_ret_5d", "HD_ret_1d", "MRK_Merck_zscore_60d"], "is_new": true}, {"model_id": "new_h3_STRESS_LightGBM_N15_t2", "algo": "LightGBM", "regime": "STRESS", "horizon": 3, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "XLY_Disc_vol_20d", "TXN_vol_20d", "MSTR_Bitcoin3_ret_20d", "MSTR_Bitcoin3_ret_5d", "DAX_Germany_zscore_60d", "Nikkei_Japan_zscore_60d", "VVIX_ret_20d", "EQIX_Equinix_ret_5d", "EWL_Switzerland_vol_20d", "EWQ_France_ret_20d", "US3M_Rate_zscore_60d", "Industrial_Production_zscore_60d", "EWQ_France_zscore_60d"], "is_new": true}, {"model_id": "new_h3_STRESS_LightGBM_N15_t3", "algo": "LightGBM", "regime": "STRESS", "horizon": 3, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "spx_abs_ret_max_5d", "LLY_zscore_60d", "LMT_LockheedMartin_ret_1d", "EWC_Canada_zscore_60d", "VVIX_ret_20d", "IYR_US_REIT2_zscore_60d", "spx_momentum_3d", "EQR_Equity_ret_1d", "heston_var_ev_h3", "MO_AltriaMG_ret_1d", "M_Macys_vol_20d", "EWG_Germany_vol_20d", "PCAR_PaccarInc_ret_5d"], "is_new": true}, {"model_id": "new_h3_STRESS_LightGBM_N15_t4", "algo": "LightGBM", "regime": "STRESS", "horizon": 3, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "LOW_Lowes_ret_20d", "EFFR_vol_20d", "IBEX_Spain_ret_20d", "LOW_Lowes_ret_5d", "TED_Spread_zscore_60d", "SBUX_zscore_60d", "ASX_Australia_ret_5d", "DAX_Germany_vol_20d", "EWL_Switzerland_zscore_60d", "CPB_CampbellSoup_ret_20d", "Core_CPI_zscore_60d", "BTI_BritishAmerican_ret_20d", "PAYX_Paychex_ret_20d"], "is_new": true}, {"model_id": "new_h3_STRESS_LightGBM_N15_t5", "algo": "LightGBM", "regime": "STRESS", "horizon": 3, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "vix_acceleration_1d", "EWG_Germany_vol_20d", "LUV_SouthwestAir_ret_5d", "US1Y_Rate_ret_5d", "EWC_Canada_zscore_60d", "NFCI_ret_5d", "DHR_vol_20d", "HUM_Humana_ret_5d", "US3M_Rate_vol_20d", "gjr_condvar_h1", "EQIX_Equinix_ret_5d", "DAX_Germany_vol_20d", "Industrial_Production_zscore_60d"], "is_new": true}, {"model_id": "new_h3_STRESS_LightGBM_N15_t6", "algo": "LightGBM", "regime": "STRESS", "horizon": 3, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "IBEX_Spain_ret_20d", "EWL_Switzerland_zscore_60d", "US1Y_Rate_ret_20d", "SCHW_Schwab_ret_5d", "T_ret_1d", "QQQ_vol_20d", "BTI_BritishAmerican_ret_5d", "PFE_ret_1d", "EWM_Malaysia_zscore_60d", "ITT_ITTInc_ret_5d", "XLF_Fin_vol_20d", "EWL_Switzerland_vol_20d", "LMT_LockheedMartin_ret_1d"], "is_new": true}, {"model_id": "new_h3_STRESS_LightGBM_N15_t7", "algo": "LightGBM", "regime": "STRESS", "horizon": 3, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "HD_ret_1d", "TM_Telephone_ret_1d", "EOG_EOGResources_vol_20d", "IYM_BasicMaterials_ret_20d", "Nikkei_Japan_zscore_60d", "EFFR_vol_20d", "SBUX_ret_5d", "EWJ_Japan_vol_20d", "XLK_Tech_zscore_60d", "NWL_Newell_ret_20d", "EWC_Canada_zscore_60d", "EWG_Germany_ret_20d", "DAX_Germany_vol_20d"], "is_new": true}, {"model_id": "new_h3_STRESS_LightGBM_N20_t0", "algo": "LightGBM", "regime": "STRESS", "horizon": 3, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "heston_var_ev_h5", "WTI_Oil_FRED_zscore_60d", "XLV_Health_zscore_60d", "DHR_vol_20d", "T_ret_1d", "NVDA_vol_20d", "spx_vol_5d", "TGT_Target_zscore_60d", "QQQ_vol_20d", "US1Y_Rate_ret_5d", "HUM_Humana_ret_5d", "EFFR_ret_1d", "vix_acceleration_1d", "PPL_PPL_ret_1d", "SBUX_ret_5d", "IYM_BasicMaterials_ret_20d", "CPB_CampbellSoup_vol_20d", "INTC_ret_1d"], "is_new": true}, {"model_id": "new_h3_STRESS_LightGBM_N20_t1", "algo": "LightGBM", "regime": "STRESS", "horizon": 3, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "AXP_Amex_vol_20d", "VRP_ma5", "ORCL_zscore_60d", "LMT_LockheedMartin_vol_20d", "ASX_Australia_vol_20d", "MSTR_Bitcoin3_ret_5d", "MSTR_Bitcoin3_ret_1d", "XLV_Health_zscore_60d", "CPB_CampbellSoup_vol_20d", "heston_var_ev_h5", "DAX_Germany_zscore_60d", "TXN_vol_20d", "INTC_ret_1d", "MS_MorganStanley_zscore_60d", "SPY_zscore_60d", "IYM_BasicMaterials_ret_20d", "US3M_Rate_vol_20d", "GILD_Gilead_ret_20d"], "is_new": true}, {"model_id": "new_h3_STRESS_LightGBM_N20_t2", "algo": "LightGBM", "regime": "STRESS", "horizon": 3, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "US3Y_Rate_ret_5d", "EWQ_France_ret_20d", "EWG_Germany_vol_20d", "CPB_CampbellSoup_ret_5d", "CI_Cigna_vol_20d", "CLX_Clorox_vol_20d", "INTC_ret_5d", "MSTR_Bitcoin3_ret_20d", "heston_var_ev_h3", "TM_Telephone_vol_20d", "IBEX_Spain_ret_20d", "AORD_AUS_zscore_60d", "EOG_EOGResources_ret_5d", "MS_MorganStanley_ret_1d", "DAX_Germany_zscore_60d", "SPY_zscore_60d", "AVB_AvalonBay_zscore_60d", "MS_MorganStanley_ret_5d"], "is_new": true}, {"model_id": "new_h3_STRESS_LightGBM_N20_t3", "algo": "LightGBM", "regime": "STRESS", "horizon": 3, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWY_Korea_ret_20d", "EOG_EOGResources_ret_5d", "EWM_Malaysia_zscore_60d", "MO_AltriaMG_ret_1d", "DAX_Germany_vol_20d", "T_ret_1d", "IWM_SmallCap_vol_20d", "DIS_vol_20d", "VVIX_ret_20d", "spx_momentum_3d", "CLX_Clorox_vol_20d", "Nikkei_Japan_vol_20d", "LMT_LockheedMartin_vol_20d", "M_Macys_vol_20d", "CMCSA_ret_1d", "EWG_Germany_ret_20d", "BLK_BlackRock_zscore_60d", "ORCL_zscore_60d"], "is_new": true}, {"model_id": "new_h3_STRESS_LightGBM_N20_t4", "algo": "LightGBM", "regime": "STRESS", "horizon": 3, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "heston_var_ev_h7", "vix_mean_abs_ret_5d", "HD_zscore_60d", "SLB_Schlumberger_ret_5d", "ITT_ITTInc_ret_5d", "GE_ret_1d", "HangSeng_HK_vol_20d", "EWC_Canada_zscore_60d", "AORD_AUS_zscore_60d", "3M_vol_20d", "PG_ret_20d", "Industrial_Production_zscore_60d", "PAYX_Paychex_zscore_60d", "SBUX_zscore_60d", "PPL_PPL_ret_1d", "DAX_Germany_vol_20d", "HUM_Humana_ret_5d", "NVDA_vol_20d"], "is_new": true}, {"model_id": "new_h3_STRESS_LightGBM_N20_t5", "algo": "LightGBM", "regime": "STRESS", "horizon": 3, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "CPB_CampbellSoup_ret_20d", "M_Macys_vol_20d", "EWS_Singapore_ret_5d", "EWG_Germany_ret_20d", "EOG_EOGResources_vol_20d", "LUV_SouthwestAir_ret_5d", "EWL_Switzerland_zscore_60d", "Nikkei_Japan_zscore_60d", "EWC_Canada_zscore_60d", "AVB_AvalonBay_zscore_60d", "IYR_US_REIT2_zscore_60d", "PCAR_PaccarInc_ret_5d", "LMT_LockheedMartin_vol_20d", "PG_ret_20d", "SLB_Schlumberger_ret_1d", "MSTR_Bitcoin3_ret_20d", "Michigan_Sentiment_ret_20d", "BDX_Becton_Dickinson_ret_20d"], "is_new": true}, {"model_id": "new_h3_STRESS_LightGBM_N20_t6", "algo": "LightGBM", "regime": "STRESS", "horizon": 3, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "AMT_AmericanTower_ret_1d", "PCAR_PaccarInc_ret_5d", "MSTR_Bitcoin3_ret_20d", "Nikkei_Japan_vol_20d", "AXP_Amex_ret_20d", "MSTR_Bitcoin3_ret_5d", "US5Y_Rate_ret_5d", "heston_var_ev_h5", "LOW_Lowes_ret_20d", "AMGN_Amgen_ret_1d", "SO_SouthernCo_ret_5d", "LOW_Lowes_ret_5d", "DIS_vol_20d", "CCI_CrownCastle_vol_20d", "AXP_Amex_vol_20d", "NVDA_vol_20d", "XLV_Health_zscore_60d", "NEE_NextEra_ret_20d"], "is_new": true}, {"model_id": "new_h3_STRESS_LightGBM_N20_t7", "algo": "LightGBM", "regime": "STRESS", "horizon": 3, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "MSTR_Bitcoin3_ret_5d", "EXC_Exelon_ret_1d", "SJM_JM_Smucker_ret_1d", "GE_ret_1d", "CPB_CampbellSoup_zscore_60d", "hmm_p_stress", "EXC_Exelon_zscore_60d", "BTI_BritishAmerican_ret_20d", "EQR_Equity_ret_1d", "AMD_ret_5d", "EWC_Canada_zscore_60d", "US3M_Rate_zscore_60d", "ORCL_vol_20d", "T10Y2Y_Spread_ret_5d", "HD_ret_20d", "SO_SouthernCo_ret_5d", "EWH_HongKong_ret_5d", "INTC_ret_5d"], "is_new": true}, {"model_id": "new_h3_STRESS_LightGBM_N25_t0", "algo": "LightGBM", "regime": "STRESS", "horizon": 3, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "3M_vol_20d", "BTI_BritishAmerican_ret_5d", "US1Y_Rate_ret_20d", "spx_momentum_3d", "EFFR_ret_1d", "vix_mean_abs_ret_5d", "SO_SouthernCo_ret_5d", "SBUX_vol_20d", "CPB_CampbellSoup_zscore_60d", "MS_MorganStanley_ret_1d", "gjr_condvar_h1", "EWL_Switzerland_vol_20d", "TED_Spread_zscore_60d", "spx_vol_5d", "DAX_Germany_vol_20d", "VRP_ma5", "XOM_ret_20d", "US6M_Rate_ret_20d", "DIS_vol_20d", "Retail_Sales_zscore_60d", "CCI_CrownCastle_vol_20d", "US3M_Rate_vol_20d", "CPB_CampbellSoup_ret_20d"], "is_new": true}, {"model_id": "new_h3_STRESS_LightGBM_N25_t1", "algo": "LightGBM", "regime": "STRESS", "horizon": 3, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "XLB_Materials_zscore_60d", "DAX_Germany_zscore_60d", "CPB_CampbellSoup_ret_20d", "NFCI_ret_5d", "IYR_US_REIT2_zscore_60d", "3M_ret_5d", "spx_vol_5d", "TED_Spread_vol_20d", "Brent_Oil_FRED_ret_20d", "PLD_Prologis_ret_5d", "LOW_Lowes_ret_5d", "PCAR_PaccarInc_ret_5d", "NVDA_vol_20d", "AMD_ret_1d", "CPB_CampbellSoup_vol_20d", "SLB_Schlumberger_ret_5d", "ASX_Australia_ret_5d", "T10Y2Y_Spread_ret_5d", "Nikkei_Japan_zscore_60d", "ORCL_zscore_60d", "INTC_ret_5d", "vix_acceleration_1d", "VOD_Vodafone_zscore_60d"], "is_new": true}, {"model_id": "new_h3_STRESS_LightGBM_N25_t2", "algo": "LightGBM", "regime": "STRESS", "horizon": 3, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "IYR_US_REIT2_zscore_60d", "DAX_Germany_zscore_60d", "SPY_zscore_60d", "XLV_Health_zscore_60d", "EWM_Malaysia_ret_1d", "EOG_EOGResources_ret_5d", "SLB_Schlumberger_ret_5d", "CPB_CampbellSoup_zscore_60d", "Michigan_Sentiment_ret_20d", "ORCL_vol_20d", "Industrial_Production_zscore_60d", "CCI_CrownCastle_vol_20d", "SBUX_vol_20d", "SBUX_zscore_60d", "EWG_Germany_ret_20d", "Brent_Oil_FRED_ret_5d", "EWG_Germany_vol_20d", "CI_Cigna_vol_20d", "PFE_ret_1d", "HD_zscore_60d", "US3M_Rate_zscore_60d", "EWS_Singapore_ret_5d", "Core_CPI_zscore_60d"], "is_new": true}, {"model_id": "new_h3_STRESS_LightGBM_N25_t3", "algo": "LightGBM", "regime": "STRESS", "horizon": 3, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "Nikkei_Japan_vol_20d", "EQR_Equity_ret_1d", "LLY_zscore_60d", "JNJ_ret_1d", "XLV_Health_zscore_60d", "EWY_Korea_ret_20d", "SBUX_zscore_60d", "NWL_Newell_ret_20d", "MO_AltriaMG_ret_1d", "BTI_BritishAmerican_ret_20d", "CMCSA_ret_1d", "SPY_zscore_60d", "DAX_Germany_zscore_60d", "ITT_ITTInc_ret_5d", "ORCL_vol_20d", "heston_var_ev_h5", "Industrial_Production_zscore_60d", "CTAS_Cintas_vol_20d", "MSTR_Bitcoin3_ret_5d", "EWJ_Japan_vol_20d", "XLB_Materials_zscore_60d", "PCAR_PaccarInc_ret_5d", "EOG_EOGResources_ret_5d"], "is_new": true}, {"model_id": "new_h3_STRESS_LightGBM_N25_t4", "algo": "LightGBM", "regime": "STRESS", "horizon": 3, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "TGT_Target_zscore_60d", "EWA_Australia_zscore_60d", "NFCI_ret_5d", "EOG_EOGResources_vol_20d", "vix_mean_abs_ret_5d", "CMCSA_ret_1d", "vix_acceleration_1d", "HD_ret_5d", "DAX_Germany_vol_20d", "EWM_Malaysia_zscore_60d", "IYR_US_REIT2_zscore_60d", "AMZN_ret_5d", "EWA_Australia_ret_1d", "EWG_Germany_ret_20d", "HD_ret_20d", "M_Macys_vol_20d", "AMD_ret_5d", "QQQ_vol_20d", "IBEX_Spain_ret_20d", "VRP_ma5", "NOC_Northrop_ret_20d", "SLB_Schlumberger_ret_1d", "ORCL_zscore_60d"], "is_new": true}, {"model_id": "new_h3_STRESS_LightGBM_N25_t5", "algo": "LightGBM", "regime": "STRESS", "horizon": 3, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "Nikkei_Japan_vol_20d", "INTC_ret_1d", "HangSeng_HK_ret_5d", "EWY_Korea_zscore_60d", "TM_Telephone_ret_1d", "DHR_vol_20d", "Core_CPI_zscore_60d", "SO_SouthernCo_ret_5d", "SBUX_zscore_60d", "EWL_Switzerland_vol_20d", "XLV_Health_zscore_60d", "US1Y_Rate_ret_5d", "EQIX_Equinix_ret_5d", "CPB_CampbellSoup_zscore_60d", "DAX_Germany_zscore_60d", "IYM_BasicMaterials_ret_20d", "IWM_SmallCap_vol_20d", "SLB_Schlumberger_ret_5d", "TXN_vol_20d", "heston_var_ev_h3", "PPL_PPL_ret_1d", "HD_ret_1d", "AVB_AvalonBay_zscore_60d"], "is_new": true}, {"model_id": "new_h3_STRESS_LightGBM_N25_t6", "algo": "LightGBM", "regime": "STRESS", "horizon": 3, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "JNJ_ret_1d", "BTI_BritishAmerican_ret_20d", "EWH_HongKong_ret_5d", "IBEX_Spain_ret_20d", "MSTR_Bitcoin3_ret_20d", "MRK_Merck_zscore_60d", "BA_ret_1d", "TXN_vol_20d", "NOC_Northrop_ret_20d", "ITT_ITTInc_ret_5d", "NVDA_vol_20d", "XLY_Disc_vol_20d", "US3M_Rate_vol_20d", "Nikkei_Japan_vol_20d", "EWL_Switzerland_zscore_60d", "heston_var_ev_h5", "SBUX_zscore_60d", "GD_GeneralDynamics_zscore_60d", "HUM_Humana_ret_5d", "Retail_Sales_zscore_60d", "XLF_Fin_vol_20d", "CCI_CrownCastle_vol_20d", "EWM_Malaysia_vol_20d"], "is_new": true}, {"model_id": "new_h3_STRESS_LightGBM_N25_t7", "algo": "LightGBM", "regime": "STRESS", "horizon": 3, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWQ_France_zscore_60d", "heston_var_ev_h5", "SPY_zscore_60d", "heston_var_ev_h3", "EWY_Korea_ret_20d", "FedFunds_zscore_60d", "CLX_Clorox_vol_20d", "AMZN_ret_5d", "NWL_Newell_ret_20d", "MO_AltriaMG_ret_1d", "PFE_ret_1d", "DE_Deere_vol_20d", "XLK_Tech_zscore_60d", "XLY_Disc_vol_20d", "PAYX_Paychex_vol_20d", "NVDA_vol_20d", "MSTR_Bitcoin3_ret_20d", "LUV_SouthwestAir_ret_5d", "XOM_ret_1d", "EFFR_vol_20d", "AVB_AvalonBay_zscore_60d", "WTI_Oil_FRED_zscore_60d", "SJM_JM_Smucker_ret_1d"], "is_new": true}, {"model_id": "new_h3_STRESS_LightGBM_N30_t0", "algo": "LightGBM", "regime": "STRESS", "horizon": 3, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWJ_Japan_vol_20d", "MSTR_Bitcoin3_ret_1d", "ORCL_zscore_60d", "SBUX_vol_20d", "PCAR_PaccarInc_ret_5d", "EXC_Exelon_ret_1d", "IBEX_Spain_ret_20d", "CPB_CampbellSoup_ret_20d", "ASX_Australia_ret_5d", "US30Y_Rate_ret_20d", "EWY_Korea_zscore_60d", "EWG_Germany_ret_20d", "US7Y_Rate_ret_20d", "PPL_PPL_ret_1d", "DAX_Germany_vol_20d", "CTAS_Cintas_vol_20d", "T10Y2Y_Spread_ret_5d", "ES_Evergy_ret_1d", "BDX_Becton_Dickinson_ret_20d", "AMD_ret_5d", "EWL_Switzerland_vol_20d", "GILD_Gilead_ret_20d", "ASX_Australia_vol_20d", "EWS_Singapore_ret_5d", "IYR_US_REIT2_zscore_60d", "SLB_Schlumberger_ret_5d", "EQR_Equity_ret_1d", "Retail_Sales_zscore_60d"], "is_new": true}, {"model_id": "new_h3_STRESS_LightGBM_N30_t1", "algo": "LightGBM", "regime": "STRESS", "horizon": 3, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "DE_Deere_vol_20d", "BA_ret_1d", "EWJ_Japan_vol_20d", "CCI_CrownCastle_vol_20d", "AMD_ret_1d", "ORCL_vol_20d", "MS_MorganStanley_zscore_60d", "heston_var_ev_h7", "HD_ret_20d", "hmm_p_stress", "LOW_Lowes_ret_5d", "CPB_CampbellSoup_ret_20d", "SJM_JM_Smucker_ret_5d", "GD_GeneralDynamics_zscore_60d", "QQQ_vol_20d", "SLB_Schlumberger_ret_5d", "XLK_Tech_zscore_60d", "SJM_JM_Smucker_ret_1d", "PLD_Prologis_ret_5d", "TED_Spread_vol_20d", "T_ret_1d", "SO_SouthernCo_ret_5d", "HangSeng_HK_ret_5d", "LMT_LockheedMartin_ret_1d", "XOM_ret_1d", "SBUX_zscore_60d", "LOW_Lowes_ret_20d", "WTI_Oil_FRED_zscore_60d"], "is_new": true}, {"model_id": "new_h3_STRESS_LightGBM_N30_t2", "algo": "LightGBM", "regime": "STRESS", "horizon": 3, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "HD_zscore_60d", "WTI_Oil_FRED_zscore_60d", "CLX_Clorox_vol_20d", "PFE_ret_1d", "XLB_Materials_zscore_60d", "HUM_Humana_ret_5d", "NOC_Northrop_ret_20d", "BTI_BritishAmerican_ret_5d", "XLK_Tech_zscore_60d", "BA_ret_1d", "MSTR_Bitcoin3_ret_1d", "VOD_Vodafone_zscore_60d", "EQIX_Equinix_ret_5d", "EWM_Malaysia_ret_1d", "US3Y_Rate_ret_5d", "LMT_LockheedMartin_vol_20d", "CPB_CampbellSoup_ret_20d", "AMT_AmericanTower_ret_1d", "TM_Telephone_ret_1d", "EWG_Germany_vol_20d", "DE_Deere_vol_20d", "SCHW_Schwab_ret_5d", "SPY_zscore_60d", "MS_MorganStanley_zscore_60d", "MS_MorganStanley_ret_5d", "T10Y2Y_Spread_ret_5d", "NWL_Newell_ret_20d", "TM_Telephone_vol_20d"], "is_new": true}, {"model_id": "new_h3_STRESS_LightGBM_N30_t3", "algo": "LightGBM", "regime": "STRESS", "horizon": 3, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "AMT_AmericanTower_ret_1d", "EWM_Malaysia_vol_20d", "ES_Evergy_ret_1d", "MSTR_Bitcoin3_ret_5d", "IYM_BasicMaterials_ret_20d", "BLK_BlackRock_zscore_60d", "heston_var_ev_h5", "vix_acceleration_1d", "AXP_Amex_vol_20d", "INTC_ret_5d", "DHR_ret_1d", "EXC_Exelon_zscore_60d", "CPB_CampbellSoup_ret_20d", "XOM_ret_1d", "GE_ret_1d", "EOG_EOGResources_ret_5d", "AVB_AvalonBay_zscore_60d", "PAYX_Paychex_ret_20d", "NEE_NextEra_ret_20d", "MRK_Merck_zscore_60d", "XLY_Disc_vol_20d", "EWM_Malaysia_ret_1d", "EWL_Switzerland_vol_20d", "Industrial_Production_zscore_60d", "EQIX_Equinix_ret_5d", "US3M_Rate_vol_20d", "HD_ret_20d", "VVIX_ret_20d"], "is_new": true}, {"model_id": "new_h3_STRESS_LightGBM_N30_t4", "algo": "LightGBM", "regime": "STRESS", "horizon": 3, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "gjr_condvar_h1", "US3M_Rate_zscore_60d", "NOC_Northrop_ret_20d", "Brent_Oil_FRED_ret_20d", "AORD_AUS_zscore_60d", "TM_Telephone_vol_20d", "GE_ret_1d", "vix_mean_abs_ret_5d", "vix_acceleration_1d", "ENB_EnbridgeInc_ret_1d", "VRP_ma5", "Retail_Sales_zscore_60d", "BA_ret_1d", "HangSeng_HK_ret_5d", "INTC_ret_5d", "EWG_Germany_vol_20d", "T10Y2Y_Spread_ret_5d", "NWL_Newell_ret_20d", "hmm_p_stress", "PAYX_Paychex_ret_20d", "MS_MorganStanley_zscore_60d", "BTI_BritishAmerican_ret_5d", "DHR_vol_20d", "heston_var_ev_h3", "PAYX_Paychex_vol_20d", "NFCI_ret_5d", "SBUX_zscore_60d", "MSTR_Bitcoin3_ret_1d"], "is_new": true}, {"model_id": "new_h3_STRESS_LightGBM_N30_t5", "algo": "LightGBM", "regime": "STRESS", "horizon": 3, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "HangSeng_HK_ret_1d", "gjr_condvar_h1", "ITT_ITTInc_ret_5d", "Michigan_Sentiment_ret_20d", "BTI_BritishAmerican_ret_5d", "HD_ret_1d", "EWM_Malaysia_vol_20d", "Core_PCE_zscore_60d", "LLY_zscore_60d", "EWC_Canada_zscore_60d", "SO_SouthernCo_ret_5d", "XOM_ret_20d", "EWA_Australia_ret_1d", "HangSeng_HK_ret_5d", "US1Y_Rate_ret_5d", "US5Y_Rate_ret_5d", "VOD_Vodafone_zscore_60d", "LOW_Lowes_ret_5d", "EWS_Singapore_ret_5d", "BTI_BritishAmerican_ret_20d", "EQIX_Equinix_ret_5d", "AXP_Amex_vol_20d", "ASX_Australia_ret_5d", "T10Y2Y_Spread_ret_5d", "IYM_BasicMaterials_ret_20d", "AVB_AvalonBay_zscore_60d", "IBEX_Spain_ret_20d", "XLF_Fin_vol_20d"], "is_new": true}, {"model_id": "new_h3_STRESS_LightGBM_N30_t6", "algo": "LightGBM", "regime": "STRESS", "horizon": 3, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "PAYX_Paychex_zscore_60d", "ASX_Australia_ret_5d", "EWL_Switzerland_vol_20d", "Michigan_Sentiment_ret_20d", "EWY_Korea_zscore_60d", "EQR_Equity_ret_1d", "EWS_Singapore_ret_5d", "SBUX_zscore_60d", "EWY_Korea_ret_20d", "spx_vol_5d", "Brent_Oil_FRED_ret_5d", "EWG_Germany_vol_20d", "HangSeng_HK_ret_1d", "DOW_Price_zscore_60d", "IBEX_Spain_ret_20d", "BTI_BritishAmerican_ret_20d", "HD_zscore_60d", "ORCL_zscore_60d", "DAX_Germany_vol_20d", "SO_SouthernCo_ret_5d", "US3Y_Rate_ret_5d", "SBUX_ret_5d", "MS_MorganStanley_ret_5d", "EMR_Emerson_ret_20d", "gjr_condvar_h1", "Retail_Sales_zscore_60d", "PPL_PPL_ret_1d", "HUM_Humana_ret_5d"], "is_new": true}, {"model_id": "new_h3_STRESS_LightGBM_N30_t7", "algo": "LightGBM", "regime": "STRESS", "horizon": 3, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "ASX_Australia_ret_5d", "CPB_CampbellSoup_vol_20d", "SCHW_Schwab_ret_5d", "DE_Deere_vol_20d", "spx_vol_5d", "GD_GeneralDynamics_zscore_60d", "TM_Telephone_vol_20d", "spx_momentum_3d", "CCI_CrownCastle_vol_20d", "JNJ_ret_1d", "CLX_Clorox_vol_20d", "Retail_Sales_zscore_60d", "Industrial_Production_zscore_60d", "AMT_AmericanTower_ret_1d", "SBUX_zscore_60d", "US3M_Rate_zscore_60d", "NVDA_vol_20d", "EWQ_France_ret_20d", "heston_var_ev_h3", "EWM_Malaysia_ret_1d", "DHR_vol_20d", "GILD_Gilead_ret_20d", "BLK_BlackRock_zscore_60d", "VOD_Vodafone_zscore_60d", "MS_MorganStanley_ret_1d", "WTI_Oil_FRED_zscore_60d", "XLF_Fin_vol_20d", "INTC_ret_5d"], "is_new": true}, {"model_id": "new_h3_STRESS_GradientBoosting_N5_t0", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 3, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "SBUX_ret_5d", "CMCSA_ret_1d", "GILD_Gilead_ret_20d"], "is_new": true}, {"model_id": "new_h3_STRESS_GradientBoosting_N5_t1", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 3, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "INTC_ret_5d", "TGT_Target_zscore_60d", "Nikkei_Japan_zscore_60d"], "is_new": true}, {"model_id": "new_h3_STRESS_GradientBoosting_N5_t2", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 3, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "PAYX_Paychex_ret_20d", "SLB_Schlumberger_ret_5d", "MRK_Merck_zscore_60d"], "is_new": true}, {"model_id": "new_h3_STRESS_GradientBoosting_N5_t3", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 3, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "PG_ret_20d", "DAX_Germany_vol_20d", "EQIX_Equinix_ret_5d"], "is_new": true}, {"model_id": "new_h3_STRESS_GradientBoosting_N5_t4", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 3, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "MSTR_Bitcoin3_ret_20d", "Retail_Sales_zscore_60d", "3M_ret_5d"], "is_new": true}, {"model_id": "new_h3_STRESS_GradientBoosting_N5_t5", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 3, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "DAX_Germany_zscore_60d", "EOG_EOGResources_ret_5d", "heston_var_ev_h7"], "is_new": true}, {"model_id": "new_h3_STRESS_GradientBoosting_N5_t6", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 3, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "HangSeng_HK_vol_20d", "Industrial_Production_zscore_60d", "EQR_Equity_ret_1d"], "is_new": true}, {"model_id": "new_h3_STRESS_GradientBoosting_N5_t7", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 3, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "AMD_ret_1d", "EWL_Switzerland_zscore_60d", "XLF_Fin_vol_20d"], "is_new": true}, {"model_id": "new_h3_STRESS_GradientBoosting_N8_t0", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 3, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "HD_ret_5d", "NWL_Newell_ret_20d", "JNJ_ret_1d", "US3M_Rate_vol_20d", "HangSeng_HK_vol_20d", "VVIX_ret_20d"], "is_new": true}, {"model_id": "new_h3_STRESS_GradientBoosting_N8_t1", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 3, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "GE_ret_1d", "XLB_Materials_zscore_60d", "EWA_Australia_zscore_60d", "EQIX_Equinix_ret_5d", "3M_vol_20d", "EWH_HongKong_ret_5d"], "is_new": true}, {"model_id": "new_h3_STRESS_GradientBoosting_N8_t2", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 3, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "US1Y_Rate_ret_20d", "Michigan_Sentiment_ret_20d", "SJM_JM_Smucker_ret_5d", "PPL_PPL_ret_1d", "AMZN_ret_5d", "heston_var_ev_h7"], "is_new": true}, {"model_id": "new_h3_STRESS_GradientBoosting_N8_t3", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 3, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "MRK_Merck_zscore_60d", "SBUX_zscore_60d", "TGT_Target_zscore_60d", "MO_AltriaMG_ret_1d", "HD_ret_5d", "IWM_SmallCap_vol_20d"], "is_new": true}, {"model_id": "new_h3_STRESS_GradientBoosting_N8_t4", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 3, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "3M_vol_20d", "US6M_Rate_ret_20d", "DHR_vol_20d", "EXC_Exelon_zscore_60d", "NVDA_vol_20d", "HD_ret_5d"], "is_new": true}, {"model_id": "new_h3_STRESS_GradientBoosting_N8_t5", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 3, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "DE_Deere_ret_5d", "DOW_Price_zscore_60d", "EOG_EOGResources_vol_20d", "LLY_zscore_60d", "PFE_ret_1d", "MO_AltriaMG_ret_1d"], "is_new": true}, {"model_id": "new_h3_STRESS_GradientBoosting_N8_t6", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 3, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EMR_Emerson_ret_20d", "NVDA_vol_20d", "MS_MorganStanley_ret_5d", "HangSeng_HK_vol_20d", "EWQ_France_zscore_60d", "MRK_Merck_zscore_60d"], "is_new": true}, {"model_id": "new_h3_STRESS_GradientBoosting_N8_t7", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 3, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "MO_AltriaMG_ret_1d", "heston_var_ev_h7", "heston_var_ev_h5", "PAYX_Paychex_ret_20d", "EWM_Malaysia_zscore_60d", "CPB_CampbellSoup_ret_5d"], "is_new": true}, {"model_id": "new_h3_STRESS_GradientBoosting_N10_t0", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 3, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EQIX_Equinix_ret_5d", "DIS_vol_20d", "spx_momentum_3d", "EWY_Korea_zscore_60d", "SBUX_zscore_60d", "ORCL_zscore_60d", "Nikkei_Japan_vol_20d", "3M_ret_5d"], "is_new": true}, {"model_id": "new_h3_STRESS_GradientBoosting_N10_t1", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 3, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "BTI_BritishAmerican_ret_5d", "BLK_BlackRock_zscore_60d", "AXP_Amex_vol_20d", "DHR_vol_20d", "EXC_Exelon_ret_1d", "INTC_ret_1d", "TED_Spread_vol_20d", "TED_Spread_zscore_60d"], "is_new": true}, {"model_id": "new_h3_STRESS_GradientBoosting_N10_t2", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 3, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "gjr_condvar_h1", "PFE_ret_1d", "spx_abs_ret_max_5d", "VOD_Vodafone_zscore_60d", "EWQ_France_zscore_60d", "WTI_Oil_FRED_zscore_60d", "XLB_Materials_zscore_60d", "MS_MorganStanley_ret_1d"], "is_new": true}, {"model_id": "new_h3_STRESS_GradientBoosting_N10_t3", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 3, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "XLK_Tech_zscore_60d", "EWG_Germany_ret_20d", "EWS_Singapore_ret_5d", "SLB_Schlumberger_ret_1d", "EWC_Canada_zscore_60d", "TED_Spread_vol_20d", "PAYX_Paychex_vol_20d", "Nikkei_Japan_vol_20d"], "is_new": true}, {"model_id": "new_h3_STRESS_GradientBoosting_N10_t4", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 3, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "LLY_zscore_60d", "SO_SouthernCo_ret_5d", "DAX_Germany_vol_20d", "PAYX_Paychex_vol_20d", "EWH_HongKong_ret_5d", "US1Y_Rate_ret_5d", "ORCL_vol_20d", "TM_Telephone_ret_1d"], "is_new": true}, {"model_id": "new_h3_STRESS_GradientBoosting_N10_t5", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 3, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "DHR_vol_20d", "EWY_Korea_ret_20d", "IYR_US_REIT2_zscore_60d", "ASX_Australia_vol_20d", "SBUX_zscore_60d", "heston_ev_h3", "XLB_Materials_zscore_60d", "gjr_condvar_h1"], "is_new": true}, {"model_id": "new_h3_STRESS_GradientBoosting_N10_t6", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 3, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWM_Malaysia_zscore_60d", "gjr_condvar_h1", "LOW_Lowes_ret_5d", "DE_Deere_vol_20d", "Core_CPI_zscore_60d", "US6M_Rate_ret_20d", "XLB_Materials_zscore_60d", "T10Y2Y_Spread_ret_5d"], "is_new": true}, {"model_id": "new_h3_STRESS_GradientBoosting_N10_t7", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 3, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "AORD_AUS_zscore_60d", "vix_acceleration_1d", "EWC_Canada_zscore_60d", "EWH_HongKong_ret_5d", "HD_zscore_60d", "EQIX_Equinix_ret_5d", "VVIX_ret_20d", "XLV_Health_zscore_60d"], "is_new": true}, {"model_id": "new_h3_STRESS_GradientBoosting_N12_t0", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 3, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "PG_ret_20d", "EFFR_ret_1d", "NEE_NextEra_ret_20d", "spx_vol_5d", "gjr_condvar_h1", "ORCL_vol_20d", "ES_Evergy_ret_1d", "TGT_Target_zscore_60d", "CLX_Clorox_vol_20d", "BDX_Becton_Dickinson_ret_20d"], "is_new": true}, {"model_id": "new_h3_STRESS_GradientBoosting_N12_t1", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 3, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "PAYX_Paychex_ret_20d", "TM_Telephone_ret_1d", "NFCI_ret_5d", "vix_mean_abs_ret_5d", "EWM_Malaysia_vol_20d", "XLB_Materials_zscore_60d", "SBUX_ret_5d", "Brent_Oil_FRED_ret_5d", "TGT_Target_zscore_60d", "BTI_BritishAmerican_ret_5d"], "is_new": true}, {"model_id": "new_h3_STRESS_GradientBoosting_N12_t2", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 3, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "FedFunds_zscore_60d", "Nikkei_Japan_zscore_60d", "SJM_JM_Smucker_ret_5d", "US3Y_Rate_ret_5d", "US3M_Rate_zscore_60d", "EWM_Malaysia_vol_20d", "EWL_Switzerland_vol_20d", "ASX_Australia_ret_5d", "LOW_Lowes_ret_5d", "SBUX_vol_20d"], "is_new": true}, {"model_id": "new_h3_STRESS_GradientBoosting_N12_t3", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 3, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EXC_Exelon_ret_1d", "DHR_ret_1d", "MS_MorganStanley_ret_1d", "Core_PCE_zscore_60d", "BDX_Becton_Dickinson_ret_20d", "EWQ_France_ret_20d", "EWQ_France_zscore_60d", "DOW_Price_zscore_60d", "HD_ret_1d", "US1Y_Rate_ret_5d"], "is_new": true}, {"model_id": "new_h3_STRESS_GradientBoosting_N12_t4", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 3, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "MS_MorganStanley_ret_1d", "LMT_LockheedMartin_ret_1d", "CPB_CampbellSoup_zscore_60d", "TED_Spread_zscore_60d", "TM_Telephone_ret_1d", "GILD_Gilead_ret_20d", "XLF_Fin_vol_20d", "CTAS_Cintas_vol_20d", "XLV_Health_zscore_60d", "NEE_NextEra_ret_20d"], "is_new": true}, {"model_id": "new_h3_STRESS_GradientBoosting_N12_t5", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 3, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "US6M_Rate_ret_20d", "HangSeng_HK_ret_5d", "AMT_AmericanTower_ret_1d", "M_Macys_vol_20d", "EWQ_France_ret_20d", "LUV_SouthwestAir_ret_5d", "PPL_PPL_ret_1d", "EQR_Equity_ret_1d", "FedFunds_zscore_60d", "PLD_Prologis_ret_5d"], "is_new": true}, {"model_id": "new_h3_STRESS_GradientBoosting_N12_t6", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 3, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "DIS_vol_20d", "MS_MorganStanley_zscore_60d", "spx_vol_5d", "T10Y2Y_Spread_ret_5d", "ITT_ITTInc_ret_5d", "ASX_Australia_ret_5d", "AMD_ret_1d", "PLD_Prologis_ret_5d", "DHR_ret_1d", "US1Y_Rate_ret_20d"], "is_new": true}, {"model_id": "new_h3_STRESS_GradientBoosting_N12_t7", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 3, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "TED_Spread_vol_20d", "AMD_ret_1d", "AMZN_ret_5d", "US6M_Rate_ret_20d", "XOM_ret_1d", "VOD_Vodafone_zscore_60d", "SLB_Schlumberger_ret_5d", "Nikkei_Japan_vol_20d", "ORCL_vol_20d", "US3M_Rate_zscore_60d"], "is_new": true}, {"model_id": "new_h3_STRESS_GradientBoosting_N15_t0", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 3, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "IWM_SmallCap_vol_20d", "BTI_BritishAmerican_ret_20d", "GD_GeneralDynamics_zscore_60d", "XLV_Health_zscore_60d", "AORD_AUS_zscore_60d", "IBEX_Spain_ret_20d", "SJM_JM_Smucker_ret_5d", "PLD_Prologis_ret_5d", "EWM_Malaysia_ret_1d", "LLY_zscore_60d", "XLF_Fin_vol_20d", "SBUX_vol_20d", "MRK_Merck_zscore_60d"], "is_new": true}, {"model_id": "new_h3_STRESS_GradientBoosting_N15_t1", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 3, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWQ_France_ret_20d", "MSTR_Bitcoin3_ret_20d", "HUM_Humana_ret_5d", "XLY_Disc_vol_20d", "US3M_Rate_vol_20d", "MS_MorganStanley_ret_5d", "CPB_CampbellSoup_ret_5d", "BLK_BlackRock_zscore_60d", "SBUX_vol_20d", "SJM_JM_Smucker_ret_1d", "US3Y_Rate_ret_5d", "XLB_Materials_zscore_60d", "heston_ev_h3"], "is_new": true}, {"model_id": "new_h3_STRESS_GradientBoosting_N15_t2", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 3, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EMR_Emerson_ret_20d", "DHR_vol_20d", "Nikkei_Japan_vol_20d", "HangSeng_HK_vol_20d", "MSTR_Bitcoin3_ret_1d", "CI_Cigna_vol_20d", "ITT_ITTInc_ret_5d", "EWQ_France_zscore_60d", "EWQ_France_ret_20d", "ASX_Australia_ret_5d", "AMD_ret_1d", "DIS_vol_20d", "SJM_JM_Smucker_ret_1d"], "is_new": true}, {"model_id": "new_h3_STRESS_GradientBoosting_N15_t3", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 3, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "BLK_BlackRock_zscore_60d", "QQQ_vol_20d", "NVDA_vol_20d", "XLK_Tech_zscore_60d", "PPL_PPL_ret_1d", "ORCL_vol_20d", "US3M_Rate_vol_20d", "EWS_Singapore_ret_5d", "TM_Telephone_ret_1d", "ASX_Australia_vol_20d", "INTC_ret_5d", "CPB_CampbellSoup_zscore_60d", "EWY_Korea_zscore_60d"], "is_new": true}, {"model_id": "new_h3_STRESS_GradientBoosting_N15_t4", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 3, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "Core_CPI_zscore_60d", "ORCL_vol_20d", "SPY_zscore_60d", "Brent_Oil_FRED_ret_20d", "EFFR_ret_1d", "ASX_Australia_vol_20d", "SJM_JM_Smucker_ret_1d", "JNJ_ret_1d", "3M_vol_20d", "BA_ret_1d", "AMZN_ret_5d", "Michigan_Sentiment_ret_20d", "XLK_Tech_zscore_60d"], "is_new": true}, {"model_id": "new_h3_STRESS_GradientBoosting_N15_t5", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 3, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "TED_Spread_zscore_60d", "LLY_zscore_60d", "US3M_Rate_zscore_60d", "EWL_Switzerland_zscore_60d", "US7Y_Rate_ret_20d", "spx_vol_5d", "EFFR_vol_20d", "DHR_vol_20d", "EXC_Exelon_zscore_60d", "EXC_Exelon_ret_1d", "INTC_ret_5d", "LOW_Lowes_ret_20d", "ORCL_vol_20d"], "is_new": true}, {"model_id": "new_h3_STRESS_GradientBoosting_N15_t6", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 3, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "DAX_Germany_zscore_60d", "CPB_CampbellSoup_ret_20d", "HangSeng_HK_ret_5d", "TM_Telephone_ret_1d", "XLV_Health_zscore_60d", "IBEX_Spain_ret_20d", "HD_ret_1d", "CI_Cigna_vol_20d", "EWA_Australia_ret_1d", "XOM_ret_1d", "US3Y_Rate_ret_5d", "US5Y_Rate_ret_5d", "MRK_Merck_zscore_60d"], "is_new": true}, {"model_id": "new_h3_STRESS_GradientBoosting_N15_t7", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 3, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "US3M_Rate_vol_20d", "WTI_Oil_FRED_zscore_60d", "EWC_Canada_zscore_60d", "EWH_HongKong_ret_5d", "BA_ret_1d", "EWQ_France_zscore_60d", "US1Y_Rate_ret_20d", "CCI_CrownCastle_vol_20d", "INTC_ret_5d", "ORCL_vol_20d", "US7Y_Rate_ret_20d", "IWM_SmallCap_vol_20d", "VOD_Vodafone_zscore_60d"], "is_new": true}, {"model_id": "new_h3_STRESS_GradientBoosting_N20_t0", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 3, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "spx_momentum_3d", "ORCL_vol_20d", "WTI_Oil_FRED_zscore_60d", "MO_AltriaMG_ret_1d", "BTI_BritishAmerican_ret_20d", "AMD_ret_5d", "Nikkei_Japan_zscore_60d", "EWH_HongKong_ret_5d", "NFCI_ret_5d", "CTAS_Cintas_vol_20d", "hmm_p_stress", "DHR_ret_1d", "SPY_zscore_60d", "SBUX_vol_20d", "HD_zscore_60d", "AVB_AvalonBay_zscore_60d", "SJM_JM_Smucker_ret_5d", "heston_var_ev_h7"], "is_new": true}, {"model_id": "new_h3_STRESS_GradientBoosting_N20_t1", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 3, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "AMZN_ret_5d", "MSTR_Bitcoin3_ret_1d", "US5Y_Rate_ret_5d", "WTI_Oil_FRED_zscore_60d", "EWM_Malaysia_vol_20d", "TED_Spread_zscore_60d", "DIS_vol_20d", "BTI_BritishAmerican_ret_5d", "GILD_Gilead_ret_20d", "INTC_ret_1d", "DE_Deere_ret_5d", "TXN_vol_20d", "ORCL_zscore_60d", "SPY_zscore_60d", "US7Y_Rate_ret_20d", "XLB_Materials_zscore_60d", "US6M_Rate_ret_20d", "US3M_Rate_vol_20d"], "is_new": true}, {"model_id": "new_h3_STRESS_GradientBoosting_N20_t2", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 3, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "heston_var_ev_h5", "ES_Evergy_ret_1d", "US1Y_Rate_ret_20d", "HD_ret_5d", "BDX_Becton_Dickinson_ret_20d", "LOW_Lowes_ret_5d", "DHR_vol_20d", "Brent_Oil_FRED_ret_5d", "SJM_JM_Smucker_ret_5d", "LOW_Lowes_ret_20d", "AXP_Amex_vol_20d", "ITT_ITTInc_ret_5d", "MSTR_Bitcoin3_ret_5d", "BA_ret_1d", "US30Y_Rate_ret_20d", "EWA_Australia_ret_1d", "CPB_CampbellSoup_zscore_60d", "US3Y_Rate_ret_5d"], "is_new": true}, {"model_id": "new_h3_STRESS_GradientBoosting_N20_t3", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 3, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "FedFunds_zscore_60d", "TXN_vol_20d", "LMT_LockheedMartin_vol_20d", "TED_Spread_zscore_60d", "US3M_Rate_zscore_60d", "NVDA_vol_20d", "IWM_SmallCap_vol_20d", "XLK_Tech_zscore_60d", "US1Y_Rate_ret_20d", "BTI_BritishAmerican_ret_5d", "T_ret_1d", "GE_ret_1d", "AMD_ret_5d", "VVIX_ret_20d", "EWQ_France_zscore_60d", "Nikkei_Japan_vol_20d", "HangSeng_HK_vol_20d", "PFE_ret_1d"], "is_new": true}, {"model_id": "new_h3_STRESS_GradientBoosting_N20_t4", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 3, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "MRK_Merck_zscore_60d", "EWY_Korea_zscore_60d", "SLB_Schlumberger_ret_5d", "PFE_ret_1d", "HangSeng_HK_vol_20d", "TED_Spread_zscore_60d", "PPL_PPL_ret_1d", "ASX_Australia_vol_20d", "Industrial_Production_zscore_60d", "M_Macys_vol_20d", "MSTR_Bitcoin3_ret_20d", "EFFR_ret_1d", "EWM_Malaysia_ret_1d", "AORD_AUS_zscore_60d", "AXP_Amex_vol_20d", "MS_MorganStanley_ret_5d", "3M_vol_20d", "US7Y_Rate_ret_20d"], "is_new": true}, {"model_id": "new_h3_STRESS_GradientBoosting_N20_t5", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 3, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "CPB_CampbellSoup_zscore_60d", "LOW_Lowes_ret_5d", "AMD_ret_1d", "CPB_CampbellSoup_ret_20d", "DHR_ret_1d", "EWL_Switzerland_zscore_60d", "IYR_US_REIT2_zscore_60d", "BA_ret_1d", "NOC_Northrop_ret_20d", "MS_MorganStanley_zscore_60d", "XOM_ret_20d", "MRK_Merck_zscore_60d", "MSTR_Bitcoin3_ret_1d", "Nikkei_Japan_zscore_60d", "BDX_Becton_Dickinson_ret_20d", "AORD_AUS_zscore_60d", "LMT_LockheedMartin_ret_1d", "XLY_Disc_vol_20d"], "is_new": true}, {"model_id": "new_h3_STRESS_GradientBoosting_N20_t6", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 3, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "Nikkei_Japan_vol_20d", "IYM_BasicMaterials_ret_20d", "SBUX_vol_20d", "TM_Telephone_ret_1d", "US5Y_Rate_ret_5d", "EWJ_Japan_vol_20d", "QQQ_vol_20d", "US3Y_Rate_ret_5d", "EWY_Korea_ret_20d", "US7Y_Rate_ret_20d", "CPB_CampbellSoup_ret_20d", "XLB_Materials_zscore_60d", "HangSeng_HK_vol_20d", "GD_GeneralDynamics_zscore_60d", "EWM_Malaysia_ret_1d", "PG_ret_20d", "DOW_Price_zscore_60d", "US3M_Rate_zscore_60d"], "is_new": true}, {"model_id": "new_h3_STRESS_GradientBoosting_N20_t7", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 3, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EFFR_ret_1d", "CI_Cigna_vol_20d", "US3M_Rate_zscore_60d", "MS_MorganStanley_ret_1d", "AMGN_Amgen_ret_1d", "VRP_ma5", "DAX_Germany_vol_20d", "AMZN_ret_5d", "DHR_ret_1d", "PCAR_PaccarInc_ret_5d", "NWL_Newell_ret_20d", "GE_ret_1d", "SJM_JM_Smucker_ret_1d", "Brent_Oil_FRED_ret_20d", "HangSeng_HK_vol_20d", "ES_Evergy_ret_1d", "EWY_Korea_zscore_60d", "hmm_p_stress"], "is_new": true}, {"model_id": "new_h3_STRESS_GradientBoosting_N25_t0", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 3, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "Brent_Oil_FRED_ret_5d", "gjr_condvar_h1", "US6M_Rate_ret_20d", "AXP_Amex_vol_20d", "DE_Deere_ret_5d", "ORCL_vol_20d", "heston_var_ev_h7", "NOC_Northrop_ret_20d", "DE_Deere_vol_20d", "EWQ_France_ret_20d", "SBUX_vol_20d", "PAYX_Paychex_zscore_60d", "EWA_Australia_zscore_60d", "US1Y_Rate_ret_20d", "CPB_CampbellSoup_zscore_60d", "EWL_Switzerland_zscore_60d", "EWY_Korea_ret_20d", "3M_vol_20d", "TM_Telephone_ret_1d", "EWS_Singapore_ret_5d", "CPB_CampbellSoup_ret_5d", "FedFunds_zscore_60d", "MO_AltriaMG_ret_1d"], "is_new": true}, {"model_id": "new_h3_STRESS_GradientBoosting_N25_t1", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 3, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWH_HongKong_ret_5d", "IBEX_Spain_ret_20d", "MSTR_Bitcoin3_ret_5d", "EWL_Switzerland_vol_20d", "EWC_Canada_zscore_60d", "EQR_Equity_ret_1d", "SO_SouthernCo_ret_5d", "PAYX_Paychex_ret_20d", "FedFunds_zscore_60d", "heston_ev_h3", "EFFR_ret_1d", "LUV_SouthwestAir_ret_5d", "NOC_Northrop_ret_20d", "INTC_ret_5d", "Michigan_Sentiment_ret_20d", "US1Y_Rate_ret_20d", "MS_MorganStanley_ret_5d", "NFCI_ret_5d", "TGT_Target_zscore_60d", "ASX_Australia_ret_5d", "IYR_US_REIT2_zscore_60d", "XLK_Tech_zscore_60d", "AVB_AvalonBay_zscore_60d"], "is_new": true}, {"model_id": "new_h3_STRESS_GradientBoosting_N25_t2", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 3, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWL_Switzerland_vol_20d", "TXN_vol_20d", "Core_CPI_zscore_60d", "IWM_SmallCap_vol_20d", "PPL_PPL_ret_1d", "MRK_Merck_zscore_60d", "BA_ret_1d", "CMCSA_ret_1d", "INTC_ret_1d", "GE_ret_1d", "EWL_Switzerland_zscore_60d", "vix_acceleration_1d", "IYR_US_REIT2_zscore_60d", "PCAR_PaccarInc_ret_5d", "TM_Telephone_vol_20d", "CPB_CampbellSoup_zscore_60d", "US30Y_Rate_ret_20d", "AXP_Amex_vol_20d", "SBUX_ret_5d", "XLY_Disc_vol_20d", "GILD_Gilead_ret_20d", "XLK_Tech_zscore_60d", "HD_ret_20d"], "is_new": true}, {"model_id": "new_h3_STRESS_GradientBoosting_N25_t3", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 3, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "HUM_Humana_ret_5d", "GE_ret_1d", "Nikkei_Japan_vol_20d", "NEE_NextEra_ret_20d", "CMCSA_ret_1d", "VVIX_ret_20d", "hmm_p_stress", "JNJ_ret_1d", "EWY_Korea_zscore_60d", "CTAS_Cintas_vol_20d", "PCAR_PaccarInc_ret_5d", "spx_abs_ret_max_5d", "ASX_Australia_vol_20d", "3M_ret_5d", "spx_momentum_3d", "LUV_SouthwestAir_ret_5d", "NOC_Northrop_ret_20d", "DHR_vol_20d", "MS_MorganStanley_zscore_60d", "MS_MorganStanley_ret_5d", "spx_vol_5d", "TM_Telephone_ret_1d", "US1Y_Rate_ret_20d"], "is_new": true}, {"model_id": "new_h3_STRESS_GradientBoosting_N25_t4", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 3, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWQ_France_ret_20d", "heston_ev_h3", "M_Macys_vol_20d", "EWL_Switzerland_zscore_60d", "SJM_JM_Smucker_ret_1d", "spx_abs_ret_max_5d", "NWL_Newell_ret_20d", "Nikkei_Japan_zscore_60d", "LOW_Lowes_ret_5d", "XLB_Materials_zscore_60d", "XLY_Disc_vol_20d", "LMT_LockheedMartin_vol_20d", "gjr_condvar_h1", "WTI_Oil_FRED_zscore_60d", "AXP_Amex_ret_20d", "EOG_EOGResources_ret_5d", "EWM_Malaysia_vol_20d", "GILD_Gilead_ret_20d", "HD_ret_20d", "DE_Deere_ret_5d", "EWM_Malaysia_zscore_60d", "US30Y_Rate_ret_20d", "CPB_CampbellSoup_vol_20d"], "is_new": true}, {"model_id": "new_h3_STRESS_GradientBoosting_N25_t5", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 3, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "Nikkei_Japan_vol_20d", "GD_GeneralDynamics_zscore_60d", "SLB_Schlumberger_ret_5d", "IBEX_Spain_ret_20d", "XLV_Health_zscore_60d", "heston_var_ev_h7", "CPB_CampbellSoup_zscore_60d", "TED_Spread_vol_20d", "MS_MorganStanley_ret_5d", "CPB_CampbellSoup_ret_20d", "spx_momentum_3d", "MSTR_Bitcoin3_ret_5d", "EMR_Emerson_ret_20d", "T_ret_1d", "SPY_zscore_60d", "EWM_Malaysia_zscore_60d", "Michigan_Sentiment_ret_20d", "EWL_Switzerland_vol_20d", "IYR_US_REIT2_zscore_60d", "TM_Telephone_ret_1d", "BLK_BlackRock_zscore_60d", "MRK_Merck_zscore_60d", "EWG_Germany_vol_20d"], "is_new": true}, {"model_id": "new_h3_STRESS_GradientBoosting_N25_t6", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 3, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "gjr_condvar_h1", "EWG_Germany_ret_20d", "Brent_Oil_FRED_ret_5d", "EWC_Canada_zscore_60d", "LLY_zscore_60d", "Nikkei_Japan_zscore_60d", "Core_CPI_zscore_60d", "CPB_CampbellSoup_zscore_60d", "PAYX_Paychex_ret_20d", "3M_vol_20d", "EMR_Emerson_ret_20d", "SBUX_vol_20d", "SLB_Schlumberger_ret_1d", "HD_zscore_60d", "QQQ_vol_20d", "EWA_Australia_ret_1d", "HangSeng_HK_ret_1d", "HangSeng_HK_vol_20d", "PCAR_PaccarInc_ret_5d", "US5Y_Rate_ret_5d", "EWA_Australia_zscore_60d", "PAYX_Paychex_vol_20d", "EOG_EOGResources_ret_5d"], "is_new": true}, {"model_id": "new_h3_STRESS_GradientBoosting_N25_t7", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 3, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "SBUX_ret_5d", "MSTR_Bitcoin3_ret_20d", "Brent_Oil_FRED_ret_5d", "XLK_Tech_zscore_60d", "NEE_NextEra_ret_20d", "US1Y_Rate_ret_5d", "EMR_Emerson_ret_20d", "EOG_EOGResources_ret_5d", "LLY_zscore_60d", "T10Y2Y_Spread_ret_5d", "TXN_vol_20d", "JNJ_ret_1d", "heston_ev_h3", "GE_ret_1d", "EWC_Canada_zscore_60d", "Core_PCE_zscore_60d", "EWL_Switzerland_vol_20d", "INTC_ret_1d", "SCHW_Schwab_ret_5d", "HangSeng_HK_ret_5d", "AMD_ret_5d", "TED_Spread_zscore_60d", "PAYX_Paychex_vol_20d"], "is_new": true}, {"model_id": "new_h3_STRESS_GradientBoosting_N30_t0", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 3, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "PAYX_Paychex_zscore_60d", "EWA_Australia_zscore_60d", "XLK_Tech_zscore_60d", "Retail_Sales_zscore_60d", "DOW_Price_zscore_60d", "US5Y_Rate_ret_5d", "XOM_ret_20d", "NVDA_vol_20d", "CPB_CampbellSoup_vol_20d", "SBUX_vol_20d", "LLY_zscore_60d", "NEE_NextEra_ret_20d", "JNJ_ret_1d", "ORCL_zscore_60d", "heston_ev_h3", "VOD_Vodafone_zscore_60d", "US3Y_Rate_ret_5d", "BDX_Becton_Dickinson_ret_20d", "MSTR_Bitcoin3_ret_5d", "DAX_Germany_vol_20d", "SLB_Schlumberger_ret_1d", "EWS_Singapore_ret_5d", "HD_zscore_60d", "FedFunds_zscore_60d", "spx_abs_ret_max_5d", "IYM_BasicMaterials_ret_20d", "VVIX_ret_20d", "CLX_Clorox_vol_20d"], "is_new": true}, {"model_id": "new_h3_STRESS_GradientBoosting_N30_t1", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 3, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "IYR_US_REIT2_zscore_60d", "INTC_ret_1d", "Brent_Oil_FRED_ret_5d", "US7Y_Rate_ret_20d", "DAX_Germany_zscore_60d", "HD_zscore_60d", "CPB_CampbellSoup_vol_20d", "MO_AltriaMG_ret_1d", "DE_Deere_ret_5d", "SO_SouthernCo_ret_5d", "Nikkei_Japan_zscore_60d", "EWQ_France_ret_20d", "IBEX_Spain_ret_20d", "T_ret_1d", "EWY_Korea_zscore_60d", "3M_vol_20d", "BTI_BritishAmerican_ret_5d", "Nikkei_Japan_vol_20d", "CLX_Clorox_vol_20d", "PAYX_Paychex_zscore_60d", "AXP_Amex_vol_20d", "MSTR_Bitcoin3_ret_20d", "XLV_Health_zscore_60d", "Core_PCE_zscore_60d", "HangSeng_HK_ret_5d", "HangSeng_HK_vol_20d", "IWM_SmallCap_vol_20d", "EWJ_Japan_vol_20d"], "is_new": true}, {"model_id": "new_h3_STRESS_GradientBoosting_N30_t2", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 3, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "AMT_AmericanTower_ret_1d", "CPB_CampbellSoup_ret_20d", "LOW_Lowes_ret_5d", "MSTR_Bitcoin3_ret_20d", "XLK_Tech_zscore_60d", "vix_acceleration_1d", "GILD_Gilead_ret_20d", "ORCL_vol_20d", "CPB_CampbellSoup_zscore_60d", "spx_momentum_3d", "HangSeng_HK_ret_1d", "Brent_Oil_FRED_ret_5d", "LMT_LockheedMartin_ret_1d", "QQQ_vol_20d", "EWM_Malaysia_vol_20d", "EWA_Australia_ret_1d", "XLY_Disc_vol_20d", "NWL_Newell_ret_20d", "SPY_zscore_60d", "GE_ret_1d", "3M_vol_20d", "CLX_Clorox_vol_20d", "EWG_Germany_ret_20d", "GD_GeneralDynamics_zscore_60d", "Brent_Oil_FRED_ret_20d", "SBUX_zscore_60d", "SLB_Schlumberger_ret_1d", "BTI_BritishAmerican_ret_5d"], "is_new": true}, {"model_id": "new_h3_STRESS_GradientBoosting_N30_t3", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 3, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "US3Y_Rate_ret_5d", "MRK_Merck_zscore_60d", "SBUX_ret_5d", "heston_var_ev_h5", "CPB_CampbellSoup_vol_20d", "HangSeng_HK_ret_1d", "LLY_zscore_60d", "spx_abs_ret_max_5d", "IBEX_Spain_ret_20d", "US7Y_Rate_ret_20d", "LOW_Lowes_ret_20d", "CPB_CampbellSoup_ret_5d", "HD_ret_1d", "heston_ev_h3", "gjr_condvar_h1", "IWM_SmallCap_vol_20d", "TGT_Target_zscore_60d", "EWA_Australia_ret_1d", "TM_Telephone_vol_20d", "SBUX_vol_20d", "EWJ_Japan_vol_20d", "MSTR_Bitcoin3_ret_5d", "T_ret_1d", "ASX_Australia_vol_20d", "PLD_Prologis_ret_5d", "EOG_EOGResources_vol_20d", "MS_MorganStanley_ret_5d", "Nikkei_Japan_vol_20d"], "is_new": true}, {"model_id": "new_h3_STRESS_GradientBoosting_N30_t4", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 3, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWQ_France_zscore_60d", "T10Y2Y_Spread_ret_5d", "SPY_zscore_60d", "BTI_BritishAmerican_ret_5d", "MSTR_Bitcoin3_ret_20d", "TM_Telephone_vol_20d", "IBEX_Spain_ret_20d", "TED_Spread_zscore_60d", "DHR_ret_1d", "PLD_Prologis_ret_5d", "EOG_EOGResources_ret_5d", "EWC_Canada_zscore_60d", "AMT_AmericanTower_ret_1d", "VVIX_ret_20d", "XLB_Materials_zscore_60d", "ASX_Australia_ret_5d", "PCAR_PaccarInc_ret_5d", "AMGN_Amgen_ret_1d", "heston_var_ev_h7", "MSTR_Bitcoin3_ret_5d", "GILD_Gilead_ret_20d", "EWY_Korea_zscore_60d", "JNJ_ret_1d", "3M_ret_5d", "EWQ_France_ret_20d", "XOM_ret_1d", "3M_vol_20d", "MSTR_Bitcoin3_ret_1d"], "is_new": true}, {"model_id": "new_h3_STRESS_GradientBoosting_N30_t5", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 3, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "DAX_Germany_vol_20d", "XLK_Tech_zscore_60d", "XLV_Health_zscore_60d", "AXP_Amex_vol_20d", "SLB_Schlumberger_ret_5d", "CI_Cigna_vol_20d", "SO_SouthernCo_ret_5d", "Retail_Sales_zscore_60d", "EWM_Malaysia_zscore_60d", "EWJ_Japan_vol_20d", "BA_ret_1d", "AORD_AUS_zscore_60d", "EQIX_Equinix_ret_5d", "SCHW_Schwab_ret_5d", "MSTR_Bitcoin3_ret_1d", "EWQ_France_zscore_60d", "DHR_ret_1d", "PCAR_PaccarInc_ret_5d", "EQR_Equity_ret_1d", "ENB_EnbridgeInc_ret_1d", "EWG_Germany_vol_20d", "SJM_JM_Smucker_ret_1d", "EWL_Switzerland_zscore_60d", "Nikkei_Japan_vol_20d", "ITT_ITTInc_ret_5d", "BLK_BlackRock_zscore_60d", "FedFunds_zscore_60d", "EWH_HongKong_ret_5d"], "is_new": true}, {"model_id": "new_h3_STRESS_GradientBoosting_N30_t6", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 3, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "US7Y_Rate_ret_20d", "AXP_Amex_vol_20d", "EXC_Exelon_zscore_60d", "PG_ret_20d", "EWM_Malaysia_zscore_60d", "LOW_Lowes_ret_20d", "EWH_HongKong_ret_5d", "IWM_SmallCap_vol_20d", "DHR_vol_20d", "ENB_EnbridgeInc_ret_1d", "NVDA_vol_20d", "LUV_SouthwestAir_ret_5d", "spx_vol_5d", "VOD_Vodafone_zscore_60d", "EWG_Germany_vol_20d", "BA_ret_1d", "LLY_zscore_60d", "heston_var_ev_h7", "US3M_Rate_vol_20d", "PLD_Prologis_ret_5d", "EWA_Australia_ret_1d", "HangSeng_HK_vol_20d", "MO_AltriaMG_ret_1d", "CPB_CampbellSoup_ret_20d", "EWA_Australia_zscore_60d", "XLF_Fin_vol_20d", "ORCL_vol_20d", "HUM_Humana_ret_5d"], "is_new": true}, {"model_id": "new_h3_STRESS_GradientBoosting_N30_t7", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 3, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "BLK_BlackRock_zscore_60d", "AMZN_ret_5d", "XLK_Tech_zscore_60d", "US30Y_Rate_ret_20d", "XOM_ret_1d", "PFE_ret_1d", "EFFR_vol_20d", "PLD_Prologis_ret_5d", "EXC_Exelon_ret_1d", "AMD_ret_1d", "INTC_ret_1d", "CLX_Clorox_vol_20d", "EXC_Exelon_zscore_60d", "EWS_Singapore_ret_5d", "Core_CPI_zscore_60d", "MS_MorganStanley_ret_1d", "HD_zscore_60d", "CMCSA_ret_1d", "BTI_BritishAmerican_ret_5d", "XOM_ret_20d", "EWG_Germany_ret_20d", "ORCL_vol_20d", "LUV_SouthwestAir_ret_5d", "NVDA_vol_20d", "IBEX_Spain_ret_20d", "AXP_Amex_vol_20d", "heston_var_ev_h7", "LOW_Lowes_ret_5d"], "is_new": true}, {"model_id": "new_h3_STRESS_RandomForest_N5_t0", "algo": "RandomForest", "regime": "STRESS", "horizon": 3, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "AVB_AvalonBay_zscore_60d", "NFCI_ret_5d", "DAX_Germany_zscore_60d"], "is_new": true}, {"model_id": "new_h3_STRESS_RandomForest_N5_t1", "algo": "RandomForest", "regime": "STRESS", "horizon": 3, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "T10Y2Y_Spread_ret_5d", "EXC_Exelon_zscore_60d", "EWG_Germany_vol_20d"], "is_new": true}, {"model_id": "new_h3_STRESS_RandomForest_N5_t2", "algo": "RandomForest", "regime": "STRESS", "horizon": 3, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "US3Y_Rate_ret_5d", "HangSeng_HK_ret_1d", "MO_AltriaMG_ret_1d"], "is_new": true}, {"model_id": "new_h3_STRESS_RandomForest_N5_t3", "algo": "RandomForest", "regime": "STRESS", "horizon": 3, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "BTI_BritishAmerican_ret_20d", "heston_var_ev_h5", "SBUX_ret_5d"], "is_new": true}, {"model_id": "new_h3_STRESS_RandomForest_N5_t4", "algo": "RandomForest", "regime": "STRESS", "horizon": 3, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "VVIX_ret_20d", "GE_ret_1d", "XOM_ret_1d"], "is_new": true}, {"model_id": "new_h3_STRESS_RandomForest_N5_t5", "algo": "RandomForest", "regime": "STRESS", "horizon": 3, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "LUV_SouthwestAir_ret_5d", "INTC_ret_1d", "CPB_CampbellSoup_vol_20d"], "is_new": true}, {"model_id": "new_h3_STRESS_RandomForest_N5_t6", "algo": "RandomForest", "regime": "STRESS", "horizon": 3, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "TM_Telephone_ret_1d", "EWA_Australia_ret_1d", "Nikkei_Japan_vol_20d"], "is_new": true}, {"model_id": "new_h3_STRESS_RandomForest_N5_t7", "algo": "RandomForest", "regime": "STRESS", "horizon": 3, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "PCAR_PaccarInc_ret_5d", "BDX_Becton_Dickinson_ret_20d", "EWH_HongKong_ret_5d"], "is_new": true}, {"model_id": "new_h3_STRESS_RandomForest_N8_t0", "algo": "RandomForest", "regime": "STRESS", "horizon": 3, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "DAX_Germany_zscore_60d", "XOM_ret_20d", "Brent_Oil_FRED_ret_20d", "EWS_Singapore_ret_5d", "EWM_Malaysia_ret_1d", "heston_var_ev_h5"], "is_new": true}, {"model_id": "new_h3_STRESS_RandomForest_N8_t1", "algo": "RandomForest", "regime": "STRESS", "horizon": 3, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "heston_var_ev_h5", "FedFunds_zscore_60d", "EQIX_Equinix_ret_5d", "Brent_Oil_FRED_ret_20d", "HangSeng_HK_ret_5d", "HD_ret_5d"], "is_new": true}, {"model_id": "new_h3_STRESS_RandomForest_N8_t2", "algo": "RandomForest", "regime": "STRESS", "horizon": 3, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "IYM_BasicMaterials_ret_20d", "PCAR_PaccarInc_ret_5d", "HD_ret_5d", "US3M_Rate_vol_20d", "SLB_Schlumberger_ret_1d", "TED_Spread_vol_20d"], "is_new": true}, {"model_id": "new_h3_STRESS_RandomForest_N8_t3", "algo": "RandomForest", "regime": "STRESS", "horizon": 3, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "US30Y_Rate_ret_20d", "DAX_Germany_zscore_60d", "LOW_Lowes_ret_5d", "DOW_Price_zscore_60d", "MRK_Merck_zscore_60d", "HD_ret_5d"], "is_new": true}, {"model_id": "new_h3_STRESS_RandomForest_N8_t4", "algo": "RandomForest", "regime": "STRESS", "horizon": 3, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWY_Korea_ret_20d", "Industrial_Production_zscore_60d", "EOG_EOGResources_vol_20d", "BTI_BritishAmerican_ret_20d", "LMT_LockheedMartin_vol_20d", "SBUX_zscore_60d"], "is_new": true}, {"model_id": "new_h3_STRESS_RandomForest_N8_t5", "algo": "RandomForest", "regime": "STRESS", "horizon": 3, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "AMT_AmericanTower_ret_1d", "Nikkei_Japan_zscore_60d", "TGT_Target_zscore_60d", "US3Y_Rate_ret_5d", "heston_var_ev_h3", "US3M_Rate_vol_20d"], "is_new": true}, {"model_id": "new_h3_STRESS_RandomForest_N8_t6", "algo": "RandomForest", "regime": "STRESS", "horizon": 3, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "CPB_CampbellSoup_vol_20d", "LOW_Lowes_ret_20d", "QQQ_vol_20d", "ITT_ITTInc_ret_5d", "NOC_Northrop_ret_20d", "AMGN_Amgen_ret_1d"], "is_new": true}, {"model_id": "new_h3_STRESS_RandomForest_N8_t7", "algo": "RandomForest", "regime": "STRESS", "horizon": 3, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWJ_Japan_vol_20d", "AMT_AmericanTower_ret_1d", "EMR_Emerson_ret_20d", "EQIX_Equinix_ret_5d", "CPB_CampbellSoup_vol_20d", "DHR_ret_1d"], "is_new": true}, {"model_id": "new_h3_STRESS_RandomForest_N10_t0", "algo": "RandomForest", "regime": "STRESS", "horizon": 3, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EQIX_Equinix_ret_5d", "CPB_CampbellSoup_ret_20d", "NOC_Northrop_ret_20d", "VVIX_ret_20d", "SO_SouthernCo_ret_5d", "EWM_Malaysia_ret_1d", "DE_Deere_vol_20d", "EWA_Australia_ret_1d"], "is_new": true}, {"model_id": "new_h3_STRESS_RandomForest_N10_t1", "algo": "RandomForest", "regime": "STRESS", "horizon": 3, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "FedFunds_zscore_60d", "DAX_Germany_zscore_60d", "ENB_EnbridgeInc_ret_1d", "NEE_NextEra_ret_20d", "EWM_Malaysia_zscore_60d", "XOM_ret_20d", "DE_Deere_vol_20d", "NFCI_ret_5d"], "is_new": true}, {"model_id": "new_h3_STRESS_RandomForest_N10_t2", "algo": "RandomForest", "regime": "STRESS", "horizon": 3, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "FedFunds_zscore_60d", "INTC_ret_5d", "EWG_Germany_ret_20d", "QQQ_vol_20d", "CPB_CampbellSoup_zscore_60d", "Industrial_Production_zscore_60d", "heston_var_ev_h7", "ASX_Australia_vol_20d"], "is_new": true}, {"model_id": "new_h3_STRESS_RandomForest_N10_t3", "algo": "RandomForest", "regime": "STRESS", "horizon": 3, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "SBUX_zscore_60d", "TXN_vol_20d", "ORCL_zscore_60d", "SCHW_Schwab_ret_5d", "T_ret_1d", "SJM_JM_Smucker_ret_5d", "ASX_Australia_ret_5d", "CCI_CrownCastle_vol_20d"], "is_new": true}, {"model_id": "new_h3_STRESS_RandomForest_N10_t4", "algo": "RandomForest", "regime": "STRESS", "horizon": 3, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWH_HongKong_ret_5d", "EQIX_Equinix_ret_5d", "NWL_Newell_ret_20d", "TM_Telephone_ret_1d", "HangSeng_HK_ret_1d", "ES_Evergy_ret_1d", "EWC_Canada_zscore_60d", "EOG_EOGResources_vol_20d"], "is_new": true}, {"model_id": "new_h3_STRESS_RandomForest_N10_t5", "algo": "RandomForest", "regime": "STRESS", "horizon": 3, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "LMT_LockheedMartin_vol_20d", "GD_GeneralDynamics_zscore_60d", "NOC_Northrop_ret_20d", "XLK_Tech_zscore_60d", "AVB_AvalonBay_zscore_60d", "EWH_HongKong_ret_5d", "EWA_Australia_zscore_60d", "MS_MorganStanley_ret_1d"], "is_new": true}, {"model_id": "new_h3_STRESS_RandomForest_N10_t6", "algo": "RandomForest", "regime": "STRESS", "horizon": 3, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "JNJ_ret_1d", "PAYX_Paychex_ret_20d", "CI_Cigna_vol_20d", "US3M_Rate_zscore_60d", "PCAR_PaccarInc_ret_5d", "EWY_Korea_ret_20d", "EQIX_Equinix_ret_5d", "EWG_Germany_ret_20d"], "is_new": true}, {"model_id": "new_h3_STRESS_RandomForest_N10_t7", "algo": "RandomForest", "regime": "STRESS", "horizon": 3, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "JNJ_ret_1d", "Industrial_Production_zscore_60d", "DHR_ret_1d", "PCAR_PaccarInc_ret_5d", "US30Y_Rate_ret_20d", "PPL_PPL_ret_1d", "ORCL_vol_20d", "heston_var_ev_h3"], "is_new": true}, {"model_id": "new_h3_STRESS_RandomForest_N12_t0", "algo": "RandomForest", "regime": "STRESS", "horizon": 3, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWL_Switzerland_vol_20d", "IYM_BasicMaterials_ret_20d", "Nikkei_Japan_vol_20d", "HD_ret_20d", "AMT_AmericanTower_ret_1d", "US7Y_Rate_ret_20d", "EWM_Malaysia_zscore_60d", "AMD_ret_5d", "XOM_ret_1d", "XLK_Tech_zscore_60d"], "is_new": true}, {"model_id": "new_h3_STRESS_RandomForest_N12_t1", "algo": "RandomForest", "regime": "STRESS", "horizon": 3, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "AMD_ret_5d", "SBUX_ret_5d", "US30Y_Rate_ret_20d", "ORCL_vol_20d", "SCHW_Schwab_ret_5d", "US3M_Rate_zscore_60d", "MS_MorganStanley_ret_5d", "US3Y_Rate_ret_5d", "XLF_Fin_vol_20d", "US3M_Rate_vol_20d"], "is_new": true}, {"model_id": "new_h3_STRESS_RandomForest_N12_t2", "algo": "RandomForest", "regime": "STRESS", "horizon": 3, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "US30Y_Rate_ret_20d", "PFE_ret_1d", "GE_ret_1d", "MS_MorganStanley_ret_1d", "Core_PCE_zscore_60d", "EOG_EOGResources_vol_20d", "EWQ_France_ret_20d", "spx_momentum_3d", "CI_Cigna_vol_20d", "WTI_Oil_FRED_zscore_60d"], "is_new": true}, {"model_id": "new_h3_STRESS_RandomForest_N12_t3", "algo": "RandomForest", "regime": "STRESS", "horizon": 3, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "HD_zscore_60d", "CMCSA_ret_1d", "LOW_Lowes_ret_5d", "Industrial_Production_zscore_60d", "CTAS_Cintas_vol_20d", "SCHW_Schwab_ret_5d", "IWM_SmallCap_vol_20d", "T10Y2Y_Spread_ret_5d", "MS_MorganStanley_zscore_60d", "EWL_Switzerland_vol_20d"], "is_new": true}, {"model_id": "new_h3_STRESS_RandomForest_N12_t4", "algo": "RandomForest", "regime": "STRESS", "horizon": 3, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWG_Germany_ret_20d", "ENB_EnbridgeInc_ret_1d", "QQQ_vol_20d", "ASX_Australia_ret_5d", "DE_Deere_ret_5d", "SLB_Schlumberger_ret_1d", "VRP_ma5", "XLB_Materials_zscore_60d", "M_Macys_vol_20d", "NEE_NextEra_ret_20d"], "is_new": true}, {"model_id": "new_h3_STRESS_RandomForest_N12_t5", "algo": "RandomForest", "regime": "STRESS", "horizon": 3, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWJ_Japan_vol_20d", "BTI_BritishAmerican_ret_5d", "CLX_Clorox_vol_20d", "MSTR_Bitcoin3_ret_5d", "EMR_Emerson_ret_20d", "BLK_BlackRock_zscore_60d", "vix_mean_abs_ret_5d", "Michigan_Sentiment_ret_20d", "LMT_LockheedMartin_vol_20d", "CI_Cigna_vol_20d"], "is_new": true}, {"model_id": "new_h3_STRESS_RandomForest_N12_t6", "algo": "RandomForest", "regime": "STRESS", "horizon": 3, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "XLV_Health_zscore_60d", "Michigan_Sentiment_ret_20d", "LUV_SouthwestAir_ret_5d", "HD_zscore_60d", "EWG_Germany_ret_20d", "CCI_CrownCastle_vol_20d", "AMGN_Amgen_ret_1d", "JNJ_ret_1d", "LMT_LockheedMartin_vol_20d", "ORCL_zscore_60d"], "is_new": true}, {"model_id": "new_h3_STRESS_RandomForest_N12_t7", "algo": "RandomForest", "regime": "STRESS", "horizon": 3, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "BDX_Becton_Dickinson_ret_20d", "spx_vol_5d", "DHR_vol_20d", "PG_ret_20d", "IYM_BasicMaterials_ret_20d", "US3Y_Rate_ret_5d", "NOC_Northrop_ret_20d", "AVB_AvalonBay_zscore_60d", "US1Y_Rate_ret_5d", "gjr_condvar_h1"], "is_new": true}, {"model_id": "new_h3_STRESS_RandomForest_N15_t0", "algo": "RandomForest", "regime": "STRESS", "horizon": 3, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "XLF_Fin_vol_20d", "EWA_Australia_ret_1d", "IYM_BasicMaterials_ret_20d", "T10Y2Y_Spread_ret_5d", "HangSeng_HK_ret_1d", "EWG_Germany_vol_20d", "EXC_Exelon_ret_1d", "EMR_Emerson_ret_20d", "CPB_CampbellSoup_ret_5d", "FedFunds_zscore_60d", "US3M_Rate_zscore_60d", "GILD_Gilead_ret_20d", "IYR_US_REIT2_zscore_60d"], "is_new": true}, {"model_id": "new_h3_STRESS_RandomForest_N15_t1", "algo": "RandomForest", "regime": "STRESS", "horizon": 3, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "T_ret_1d", "Nikkei_Japan_zscore_60d", "AXP_Amex_ret_20d", "NWL_Newell_ret_20d", "DHR_ret_1d", "heston_var_ev_h5", "3M_vol_20d", "XOM_ret_1d", "SO_SouthernCo_ret_5d", "CPB_CampbellSoup_zscore_60d", "US1Y_Rate_ret_20d", "GE_ret_1d", "AXP_Amex_vol_20d"], "is_new": true}, {"model_id": "new_h3_STRESS_RandomForest_N15_t2", "algo": "RandomForest", "regime": "STRESS", "horizon": 3, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "HangSeng_HK_ret_5d", "CPB_CampbellSoup_vol_20d", "HD_zscore_60d", "3M_ret_5d", "LOW_Lowes_ret_20d", "BLK_BlackRock_zscore_60d", "US1Y_Rate_ret_5d", "LMT_LockheedMartin_ret_1d", "AMD_ret_5d", "vix_acceleration_1d", "CCI_CrownCastle_vol_20d", "EWQ_France_zscore_60d", "GD_GeneralDynamics_zscore_60d"], "is_new": true}, {"model_id": "new_h3_STRESS_RandomForest_N15_t3", "algo": "RandomForest", "regime": "STRESS", "horizon": 3, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWL_Switzerland_zscore_60d", "NOC_Northrop_ret_20d", "EXC_Exelon_ret_1d", "Brent_Oil_FRED_ret_20d", "EQR_Equity_ret_1d", "EOG_EOGResources_ret_5d", "NFCI_ret_5d", "HangSeng_HK_ret_5d", "PAYX_Paychex_zscore_60d", "DE_Deere_vol_20d", "INTC_ret_5d", "GD_GeneralDynamics_zscore_60d", "US5Y_Rate_ret_5d"], "is_new": true}, {"model_id": "new_h3_STRESS_RandomForest_N15_t4", "algo": "RandomForest", "regime": "STRESS", "horizon": 3, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWJ_Japan_vol_20d", "WTI_Oil_FRED_zscore_60d", "MS_MorganStanley_zscore_60d", "AMD_ret_5d", "HD_zscore_60d", "DAX_Germany_zscore_60d", "EWY_Korea_zscore_60d", "AMZN_ret_5d", "TM_Telephone_ret_1d", "CI_Cigna_vol_20d", "PCAR_PaccarInc_ret_5d", "HD_ret_20d", "gjr_condvar_h1"], "is_new": true}, {"model_id": "new_h3_STRESS_RandomForest_N15_t5", "algo": "RandomForest", "regime": "STRESS", "horizon": 3, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "US3Y_Rate_ret_5d", "M_Macys_vol_20d", "TED_Spread_zscore_60d", "hmm_p_stress", "Industrial_Production_zscore_60d", "XOM_ret_20d", "ES_Evergy_ret_1d", "PAYX_Paychex_zscore_60d", "Nikkei_Japan_zscore_60d", "SJM_JM_Smucker_ret_5d", "EQIX_Equinix_ret_5d", "AMGN_Amgen_ret_1d", "US3M_Rate_zscore_60d"], "is_new": true}, {"model_id": "new_h3_STRESS_RandomForest_N15_t6", "algo": "RandomForest", "regime": "STRESS", "horizon": 3, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "HangSeng_HK_vol_20d", "MSTR_Bitcoin3_ret_5d", "MS_MorganStanley_ret_1d", "DAX_Germany_vol_20d", "EXC_Exelon_zscore_60d", "EFFR_vol_20d", "TM_Telephone_ret_1d", "AVB_AvalonBay_zscore_60d", "XLB_Materials_zscore_60d", "SBUX_vol_20d", "3M_ret_5d", "EMR_Emerson_ret_20d", "AMD_ret_1d"], "is_new": true}, {"model_id": "new_h3_STRESS_RandomForest_N15_t7", "algo": "RandomForest", "regime": "STRESS", "horizon": 3, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWY_Korea_ret_20d", "LOW_Lowes_ret_20d", "CPB_CampbellSoup_ret_5d", "INTC_ret_5d", "XOM_ret_1d", "Brent_Oil_FRED_ret_5d", "PPL_PPL_ret_1d", "JNJ_ret_1d", "hmm_p_stress", "EWJ_Japan_vol_20d", "EWQ_France_ret_20d", "Core_PCE_zscore_60d", "ORCL_zscore_60d"], "is_new": true}, {"model_id": "new_h3_STRESS_RandomForest_N20_t0", "algo": "RandomForest", "regime": "STRESS", "horizon": 3, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWQ_France_zscore_60d", "SJM_JM_Smucker_ret_1d", "WTI_Oil_FRED_zscore_60d", "QQQ_vol_20d", "3M_ret_5d", "TGT_Target_zscore_60d", "MS_MorganStanley_ret_1d", "BA_ret_1d", "HangSeng_HK_vol_20d", "heston_var_ev_h7", "EWQ_France_ret_20d", "PLD_Prologis_ret_5d", "Retail_Sales_zscore_60d", "LLY_zscore_60d", "T_ret_1d", "PFE_ret_1d", "EQR_Equity_ret_1d", "MO_AltriaMG_ret_1d"], "is_new": true}, {"model_id": "new_h3_STRESS_RandomForest_N20_t1", "algo": "RandomForest", "regime": "STRESS", "horizon": 3, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "DHR_ret_1d", "ASX_Australia_vol_20d", "Core_CPI_zscore_60d", "ITT_ITTInc_ret_5d", "EWM_Malaysia_vol_20d", "EWQ_France_zscore_60d", "AMZN_ret_5d", "3M_vol_20d", "EWG_Germany_ret_20d", "ES_Evergy_ret_1d", "Industrial_Production_zscore_60d", "MSTR_Bitcoin3_ret_20d", "US5Y_Rate_ret_5d", "NOC_Northrop_ret_20d", "GE_ret_1d", "PAYX_Paychex_ret_20d", "EWY_Korea_zscore_60d", "BTI_BritishAmerican_ret_5d"], "is_new": true}, {"model_id": "new_h3_STRESS_RandomForest_N20_t2", "algo": "RandomForest", "regime": "STRESS", "horizon": 3, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "LOW_Lowes_ret_20d", "CMCSA_ret_1d", "VVIX_ret_20d", "NOC_Northrop_ret_20d", "SJM_JM_Smucker_ret_5d", "EWM_Malaysia_vol_20d", "XLK_Tech_zscore_60d", "HD_zscore_60d", "TM_Telephone_vol_20d", "HangSeng_HK_ret_5d", "EFFR_vol_20d", "DHR_vol_20d", "NFCI_ret_5d", "spx_abs_ret_max_5d", "Nikkei_Japan_zscore_60d", "EWC_Canada_zscore_60d", "CLX_Clorox_vol_20d", "MO_AltriaMG_ret_1d"], "is_new": true}, {"model_id": "new_h3_STRESS_RandomForest_N20_t3", "algo": "RandomForest", "regime": "STRESS", "horizon": 3, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWG_Germany_vol_20d", "heston_var_ev_h5", "spx_abs_ret_max_5d", "IBEX_Spain_ret_20d", "US3M_Rate_zscore_60d", "Core_PCE_zscore_60d", "EWY_Korea_ret_20d", "Brent_Oil_FRED_ret_5d", "LOW_Lowes_ret_20d", "MSTR_Bitcoin3_ret_20d", "heston_var_ev_h3", "JNJ_ret_1d", "EWC_Canada_zscore_60d", "Nikkei_Japan_zscore_60d", "SJM_JM_Smucker_ret_1d", "CLX_Clorox_vol_20d", "EWH_HongKong_ret_5d", "M_Macys_vol_20d"], "is_new": true}, {"model_id": "new_h3_STRESS_RandomForest_N20_t4", "algo": "RandomForest", "regime": "STRESS", "horizon": 3, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "CI_Cigna_vol_20d", "HangSeng_HK_ret_5d", "IWM_SmallCap_vol_20d", "EOG_EOGResources_ret_5d", "ES_Evergy_ret_1d", "SO_SouthernCo_ret_5d", "heston_var_ev_h3", "DE_Deere_vol_20d", "EWJ_Japan_vol_20d", "DHR_ret_1d", "EWL_Switzerland_vol_20d", "IBEX_Spain_ret_20d", "SLB_Schlumberger_ret_5d", "spx_momentum_3d", "ITT_ITTInc_ret_5d", "VVIX_ret_20d", "PAYX_Paychex_vol_20d", "XLK_Tech_zscore_60d"], "is_new": true}, {"model_id": "new_h3_STRESS_RandomForest_N20_t5", "algo": "RandomForest", "regime": "STRESS", "horizon": 3, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "LMT_LockheedMartin_vol_20d", "ORCL_zscore_60d", "PCAR_PaccarInc_ret_5d", "INTC_ret_1d", "T_ret_1d", "EWQ_France_zscore_60d", "heston_var_ev_h5", "EWL_Switzerland_zscore_60d", "LLY_zscore_60d", "EQIX_Equinix_ret_5d", "Core_PCE_zscore_60d", "vix_mean_abs_ret_5d", "XLF_Fin_vol_20d", "SJM_JM_Smucker_ret_1d", "3M_ret_5d", "MS_MorganStanley_ret_1d", "HUM_Humana_ret_5d", "3M_vol_20d"], "is_new": true}, {"model_id": "new_h3_STRESS_RandomForest_N20_t6", "algo": "RandomForest", "regime": "STRESS", "horizon": 3, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWC_Canada_zscore_60d", "heston_var_ev_h5", "Brent_Oil_FRED_ret_5d", "ITT_ITTInc_ret_5d", "HD_ret_1d", "AVB_AvalonBay_zscore_60d", "EFFR_ret_1d", "HangSeng_HK_vol_20d", "EXC_Exelon_zscore_60d", "PFE_ret_1d", "SJM_JM_Smucker_ret_5d", "EWG_Germany_vol_20d", "AMD_ret_1d", "PAYX_Paychex_zscore_60d", "IYR_US_REIT2_zscore_60d", "AMD_ret_5d", "CPB_CampbellSoup_zscore_60d", "M_Macys_vol_20d"], "is_new": true}, {"model_id": "new_h3_STRESS_RandomForest_N20_t7", "algo": "RandomForest", "regime": "STRESS", "horizon": 3, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWA_Australia_ret_1d", "hmm_p_stress", "GD_GeneralDynamics_zscore_60d", "US3Y_Rate_ret_5d", "Core_CPI_zscore_60d", "EWY_Korea_zscore_60d", "CPB_CampbellSoup_vol_20d", "AMGN_Amgen_ret_1d", "LLY_zscore_60d", "DAX_Germany_zscore_60d", "MSTR_Bitcoin3_ret_20d", "XLB_Materials_zscore_60d", "XOM_ret_20d", "MS_MorganStanley_zscore_60d", "EWG_Germany_ret_20d", "NFCI_ret_5d", "HangSeng_HK_ret_1d", "ENB_EnbridgeInc_ret_1d"], "is_new": true}, {"model_id": "new_h3_STRESS_RandomForest_N25_t0", "algo": "RandomForest", "regime": "STRESS", "horizon": 3, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "Michigan_Sentiment_ret_20d", "EXC_Exelon_zscore_60d", "INTC_ret_1d", "HD_ret_20d", "EWG_Germany_ret_20d", "MS_MorganStanley_ret_1d", "SBUX_vol_20d", "NVDA_vol_20d", "SLB_Schlumberger_ret_5d", "SBUX_ret_5d", "XLK_Tech_zscore_60d", "Nikkei_Japan_zscore_60d", "Retail_Sales_zscore_60d", "HangSeng_HK_vol_20d", "hmm_p_stress", "XOM_ret_1d", "ES_Evergy_ret_1d", "MSTR_Bitcoin3_ret_5d", "VVIX_ret_20d", "SO_SouthernCo_ret_5d", "NEE_NextEra_ret_20d", "EWA_Australia_zscore_60d", "3M_vol_20d"], "is_new": true}, {"model_id": "new_h3_STRESS_RandomForest_N25_t1", "algo": "RandomForest", "regime": "STRESS", "horizon": 3, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "DE_Deere_ret_5d", "SBUX_vol_20d", "XLV_Health_zscore_60d", "US30Y_Rate_ret_20d", "T_ret_1d", "DE_Deere_vol_20d", "CPB_CampbellSoup_ret_20d", "PAYX_Paychex_zscore_60d", "3M_ret_5d", "EOG_EOGResources_ret_5d", "NOC_Northrop_ret_20d", "SBUX_ret_5d", "Core_PCE_zscore_60d", "XLB_Materials_zscore_60d", "US3M_Rate_zscore_60d", "MSTR_Bitcoin3_ret_5d", "Brent_Oil_FRED_ret_20d", "HUM_Humana_ret_5d", "HD_ret_20d", "heston_var_ev_h7", "TM_Telephone_vol_20d", "LMT_LockheedMartin_vol_20d", "XOM_ret_1d"], "is_new": true}, {"model_id": "new_h3_STRESS_RandomForest_N25_t2", "algo": "RandomForest", "regime": "STRESS", "horizon": 3, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "DAX_Germany_vol_20d", "NOC_Northrop_ret_20d", "AVB_AvalonBay_zscore_60d", "EWH_HongKong_ret_5d", "TXN_vol_20d", "HD_ret_20d", "US7Y_Rate_ret_20d", "IWM_SmallCap_vol_20d", "DOW_Price_zscore_60d", "MS_MorganStanley_zscore_60d", "AORD_AUS_zscore_60d", "NVDA_vol_20d", "SO_SouthernCo_ret_5d", "ES_Evergy_ret_1d", "EFFR_vol_20d", "SJM_JM_Smucker_ret_5d", "PFE_ret_1d", "US3M_Rate_vol_20d", "INTC_ret_1d", "Michigan_Sentiment_ret_20d", "Brent_Oil_FRED_ret_20d", "HangSeng_HK_vol_20d", "CI_Cigna_vol_20d"], "is_new": true}, {"model_id": "new_h3_STRESS_RandomForest_N25_t3", "algo": "RandomForest", "regime": "STRESS", "horizon": 3, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "MO_AltriaMG_ret_1d", "XLV_Health_zscore_60d", "EOG_EOGResources_ret_5d", "XLY_Disc_vol_20d", "EWA_Australia_ret_1d", "EFFR_vol_20d", "Industrial_Production_zscore_60d", "PAYX_Paychex_zscore_60d", "TED_Spread_zscore_60d", "WTI_Oil_FRED_zscore_60d", "AXP_Amex_ret_20d", "T_ret_1d", "SLB_Schlumberger_ret_1d", "IYR_US_REIT2_zscore_60d", "gjr_condvar_h1", "NWL_Newell_ret_20d", "US3M_Rate_zscore_60d", "AMD_ret_5d", "spx_abs_ret_max_5d", "BDX_Becton_Dickinson_ret_20d", "EWL_Switzerland_zscore_60d", "HD_ret_5d", "SBUX_ret_5d"], "is_new": true}, {"model_id": "new_h3_STRESS_RandomForest_N25_t4", "algo": "RandomForest", "regime": "STRESS", "horizon": 3, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "AXP_Amex_vol_20d", "DHR_ret_1d", "PPL_PPL_ret_1d", "EWJ_Japan_vol_20d", "AMZN_ret_5d", "hmm_p_stress", "vix_acceleration_1d", "EWM_Malaysia_zscore_60d", "TED_Spread_zscore_60d", "XOM_ret_20d", "LOW_Lowes_ret_5d", "US3Y_Rate_ret_5d", "IYM_BasicMaterials_ret_20d", "CPB_CampbellSoup_ret_20d", "heston_var_ev_h7", "VRP_ma5", "EFFR_ret_1d", "ASX_Australia_vol_20d", "PG_ret_20d", "XLF_Fin_vol_20d", "DIS_vol_20d", "EWC_Canada_zscore_60d", "AXP_Amex_ret_20d"], "is_new": true}, {"model_id": "new_h3_STRESS_RandomForest_N25_t5", "algo": "RandomForest", "regime": "STRESS", "horizon": 3, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWL_Switzerland_zscore_60d", "EWM_Malaysia_ret_1d", "US5Y_Rate_ret_5d", "TM_Telephone_vol_20d", "XLY_Disc_vol_20d", "LOW_Lowes_ret_20d", "BTI_BritishAmerican_ret_5d", "CPB_CampbellSoup_zscore_60d", "Core_PCE_zscore_60d", "EWQ_France_zscore_60d", "XOM_ret_20d", "spx_abs_ret_max_5d", "US7Y_Rate_ret_20d", "SJM_JM_Smucker_ret_1d", "PPL_PPL_ret_1d", "EWS_Singapore_ret_5d", "spx_momentum_3d", "PFE_ret_1d", "EFFR_vol_20d", "ORCL_zscore_60d", "US1Y_Rate_ret_20d", "PLD_Prologis_ret_5d", "MS_MorganStanley_ret_1d"], "is_new": true}, {"model_id": "new_h3_STRESS_RandomForest_N25_t6", "algo": "RandomForest", "regime": "STRESS", "horizon": 3, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "3M_vol_20d", "heston_ev_h3", "US1Y_Rate_ret_20d", "HUM_Humana_ret_5d", "EQR_Equity_ret_1d", "DE_Deere_ret_5d", "VRP_ma5", "MSTR_Bitcoin3_ret_5d", "NEE_NextEra_ret_20d", "IYM_BasicMaterials_ret_20d", "CPB_CampbellSoup_vol_20d", "HangSeng_HK_ret_5d", "EXC_Exelon_ret_1d", "EXC_Exelon_zscore_60d", "SLB_Schlumberger_ret_5d", "MSTR_Bitcoin3_ret_1d", "Industrial_Production_zscore_60d", "PFE_ret_1d", "GE_ret_1d", "EWC_Canada_zscore_60d", "AMZN_ret_5d", "Michigan_Sentiment_ret_20d", "EFFR_ret_1d"], "is_new": true}, {"model_id": "new_h3_STRESS_RandomForest_N25_t7", "algo": "RandomForest", "regime": "STRESS", "horizon": 3, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "US1Y_Rate_ret_5d", "heston_ev_h3", "HangSeng_HK_ret_1d", "US3M_Rate_vol_20d", "XLV_Health_zscore_60d", "XOM_ret_20d", "DAX_Germany_vol_20d", "VOD_Vodafone_zscore_60d", "SLB_Schlumberger_ret_1d", "ORCL_zscore_60d", "DHR_vol_20d", "AXP_Amex_vol_20d", "EOG_EOGResources_vol_20d", "VRP_ma5", "SO_SouthernCo_ret_5d", "CCI_CrownCastle_vol_20d", "BTI_BritishAmerican_ret_20d", "EWH_HongKong_ret_5d", "AMD_ret_1d", "3M_vol_20d", "SPY_zscore_60d", "GE_ret_1d", "EWQ_France_ret_20d"], "is_new": true}, {"model_id": "new_h3_STRESS_RandomForest_N30_t0", "algo": "RandomForest", "regime": "STRESS", "horizon": 3, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "CI_Cigna_vol_20d", "US6M_Rate_ret_20d", "T_ret_1d", "Core_CPI_zscore_60d", "Brent_Oil_FRED_ret_5d", "heston_ev_h3", "ORCL_zscore_60d", "VRP_ma5", "NFCI_ret_5d", "EWY_Korea_ret_20d", "EWA_Australia_zscore_60d", "AMT_AmericanTower_ret_1d", "Industrial_Production_zscore_60d", "3M_ret_5d", "PCAR_PaccarInc_ret_5d", "HD_ret_1d", "US3Y_Rate_ret_5d", "XOM_ret_1d", "BA_ret_1d", "HD_ret_20d", "VVIX_ret_20d", "EWQ_France_zscore_60d", "EOG_EOGResources_ret_5d", "heston_var_ev_h7", "CPB_CampbellSoup_ret_20d", "ENB_EnbridgeInc_ret_1d", "US1Y_Rate_ret_5d", "3M_vol_20d"], "is_new": true}, {"model_id": "new_h3_STRESS_RandomForest_N30_t1", "algo": "RandomForest", "regime": "STRESS", "horizon": 3, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "NEE_NextEra_ret_20d", "CLX_Clorox_vol_20d", "MSTR_Bitcoin3_ret_1d", "VRP_ma5", "EWJ_Japan_vol_20d", "vix_mean_abs_ret_5d", "EQIX_Equinix_ret_5d", "GE_ret_1d", "BDX_Becton_Dickinson_ret_20d", "T10Y2Y_Spread_ret_5d", "TED_Spread_vol_20d", "IYM_BasicMaterials_ret_20d", "Core_CPI_zscore_60d", "Nikkei_Japan_vol_20d", "EWM_Malaysia_zscore_60d", "DE_Deere_ret_5d", "EWQ_France_zscore_60d", "SLB_Schlumberger_ret_1d", "CPB_CampbellSoup_vol_20d", "EWC_Canada_zscore_60d", "SBUX_zscore_60d", "AMD_ret_5d", "INTC_ret_5d", "SBUX_ret_5d", "HUM_Humana_ret_5d", "Brent_Oil_FRED_ret_5d", "ASX_Australia_vol_20d", "EWS_Singapore_ret_5d"], "is_new": true}, {"model_id": "new_h3_STRESS_RandomForest_N30_t2", "algo": "RandomForest", "regime": "STRESS", "horizon": 3, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "NOC_Northrop_ret_20d", "IYR_US_REIT2_zscore_60d", "WTI_Oil_FRED_zscore_60d", "GD_GeneralDynamics_zscore_60d", "DAX_Germany_vol_20d", "IWM_SmallCap_vol_20d", "DAX_Germany_zscore_60d", "LOW_Lowes_ret_5d", "TGT_Target_zscore_60d", "EWH_HongKong_ret_5d", "TED_Spread_vol_20d", "LMT_LockheedMartin_ret_1d", "CI_Cigna_vol_20d", "Nikkei_Japan_zscore_60d", "EWM_Malaysia_zscore_60d", "INTC_ret_1d", "SCHW_Schwab_ret_5d", "SPY_zscore_60d", "Brent_Oil_FRED_ret_20d", "Core_PCE_zscore_60d", "NFCI_ret_5d", "HD_ret_20d", "HD_ret_1d", "ORCL_vol_20d", "VRP_ma5", "SJM_JM_Smucker_ret_5d", "GILD_Gilead_ret_20d", "Retail_Sales_zscore_60d"], "is_new": true}, {"model_id": "new_h3_STRESS_RandomForest_N30_t3", "algo": "RandomForest", "regime": "STRESS", "horizon": 3, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "CPB_CampbellSoup_ret_5d", "BLK_BlackRock_zscore_60d", "EQIX_Equinix_ret_5d", "XLB_Materials_zscore_60d", "hmm_p_stress", "vix_mean_abs_ret_5d", "Industrial_Production_zscore_60d", "T10Y2Y_Spread_ret_5d", "JNJ_ret_1d", "Core_PCE_zscore_60d", "spx_abs_ret_max_5d", "US1Y_Rate_ret_20d", "MO_AltriaMG_ret_1d", "TED_Spread_vol_20d", "CCI_CrownCastle_vol_20d", "heston_ev_h3", "BA_ret_1d", "CPB_CampbellSoup_ret_20d", "HD_ret_5d", "MSTR_Bitcoin3_ret_5d", "AXP_Amex_ret_20d", "DE_Deere_ret_5d", "EWA_Australia_ret_1d", "XOM_ret_1d", "EWY_Korea_ret_20d", "heston_var_ev_h7", "MS_MorganStanley_zscore_60d", "SLB_Schlumberger_ret_5d"], "is_new": true}, {"model_id": "new_h3_STRESS_RandomForest_N30_t4", "algo": "RandomForest", "regime": "STRESS", "horizon": 3, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "AXP_Amex_vol_20d", "TM_Telephone_ret_1d", "Core_CPI_zscore_60d", "DHR_ret_1d", "EWS_Singapore_ret_5d", "Industrial_Production_zscore_60d", "Nikkei_Japan_vol_20d", "EWM_Malaysia_vol_20d", "CCI_CrownCastle_vol_20d", "M_Macys_vol_20d", "SCHW_Schwab_ret_5d", "MSTR_Bitcoin3_ret_20d", "MS_MorganStanley_zscore_60d", "vix_mean_abs_ret_5d", "DHR_vol_20d", "PAYX_Paychex_zscore_60d", "CPB_CampbellSoup_ret_20d", "XOM_ret_1d", "US30Y_Rate_ret_20d", "US7Y_Rate_ret_20d", "EQR_Equity_ret_1d", "PG_ret_20d", "TED_Spread_zscore_60d", "US1Y_Rate_ret_5d", "US6M_Rate_ret_20d", "GILD_Gilead_ret_20d", "PLD_Prologis_ret_5d", "SO_SouthernCo_ret_5d"], "is_new": true}, {"model_id": "new_h3_STRESS_RandomForest_N30_t5", "algo": "RandomForest", "regime": "STRESS", "horizon": 3, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "XLV_Health_zscore_60d", "EWC_Canada_zscore_60d", "TGT_Target_zscore_60d", "EWM_Malaysia_vol_20d", "HangSeng_HK_vol_20d", "SJM_JM_Smucker_ret_1d", "PAYX_Paychex_ret_20d", "AXP_Amex_ret_20d", "LMT_LockheedMartin_ret_1d", "CLX_Clorox_vol_20d", "AMD_ret_5d", "EXC_Exelon_ret_1d", "QQQ_vol_20d", "EWM_Malaysia_zscore_60d", "3M_ret_5d", "US1Y_Rate_ret_5d", "AORD_AUS_zscore_60d", "ES_Evergy_ret_1d", "SPY_zscore_60d", "ENB_EnbridgeInc_ret_1d", "CPB_CampbellSoup_ret_20d", "EWL_Switzerland_vol_20d", "Core_CPI_zscore_60d", "CCI_CrownCastle_vol_20d", "HangSeng_HK_ret_1d", "SLB_Schlumberger_ret_5d", "EOG_EOGResources_vol_20d", "IBEX_Spain_ret_20d"], "is_new": true}, {"model_id": "new_h3_STRESS_RandomForest_N30_t6", "algo": "RandomForest", "regime": "STRESS", "horizon": 3, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "AMT_AmericanTower_ret_1d", "GILD_Gilead_ret_20d", "HUM_Humana_ret_5d", "IBEX_Spain_ret_20d", "MSTR_Bitcoin3_ret_20d", "US3M_Rate_zscore_60d", "PG_ret_20d", "TGT_Target_zscore_60d", "XOM_ret_20d", "VOD_Vodafone_zscore_60d", "spx_vol_5d", "DIS_vol_20d", "IWM_SmallCap_vol_20d", "US3M_Rate_vol_20d", "NFCI_ret_5d", "EWL_Switzerland_zscore_60d", "HangSeng_HK_ret_1d", "US6M_Rate_ret_20d", "AMGN_Amgen_ret_1d", "vix_mean_abs_ret_5d", "CMCSA_ret_1d", "PAYX_Paychex_zscore_60d", "NVDA_vol_20d", "TED_Spread_vol_20d", "EWM_Malaysia_ret_1d", "EWY_Korea_ret_20d", "vix_acceleration_1d", "LOW_Lowes_ret_5d"], "is_new": true}, {"model_id": "new_h3_STRESS_RandomForest_N30_t7", "algo": "RandomForest", "regime": "STRESS", "horizon": 3, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "XLY_Disc_vol_20d", "LUV_SouthwestAir_ret_5d", "CLX_Clorox_vol_20d", "spx_vol_5d", "EWC_Canada_zscore_60d", "gjr_condvar_h1", "AXP_Amex_ret_20d", "EWJ_Japan_vol_20d", "AMZN_ret_5d", "PAYX_Paychex_ret_20d", "MS_MorganStanley_ret_1d", "hmm_p_stress", "Brent_Oil_FRED_ret_20d", "Retail_Sales_zscore_60d", "3M_vol_20d", "XOM_ret_20d", "CPB_CampbellSoup_zscore_60d", "XLK_Tech_zscore_60d", "M_Macys_vol_20d", "US1Y_Rate_ret_5d", "GD_GeneralDynamics_zscore_60d", "NFCI_ret_5d", "IBEX_Spain_ret_20d", "TM_Telephone_ret_1d", "DE_Deere_vol_20d", "AORD_AUS_zscore_60d", "T10Y2Y_Spread_ret_5d", "SBUX_zscore_60d"], "is_new": true}, {"model_id": "new_h3_STRESS_LogisticRegression_N5_t0", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 3, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "TGT_Target_zscore_60d", "MS_MorganStanley_ret_1d", "US3Y_Rate_ret_5d"], "is_new": true}, {"model_id": "new_h3_STRESS_LogisticRegression_N5_t1", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 3, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EXC_Exelon_ret_1d", "BDX_Becton_Dickinson_ret_20d", "IWM_SmallCap_vol_20d"], "is_new": true}, {"model_id": "new_h3_STRESS_LogisticRegression_N5_t2", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 3, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "SJM_JM_Smucker_ret_1d", "PG_ret_20d", "EOG_EOGResources_ret_5d"], "is_new": true}, {"model_id": "new_h3_STRESS_LogisticRegression_N5_t3", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 3, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "hmm_p_stress", "Michigan_Sentiment_ret_20d", "EFFR_ret_1d"], "is_new": true}, {"model_id": "new_h3_STRESS_LogisticRegression_N5_t4", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 3, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "spx_vol_5d", "GE_ret_1d", "US7Y_Rate_ret_20d"], "is_new": true}, {"model_id": "new_h3_STRESS_LogisticRegression_N5_t5", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 3, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EFFR_vol_20d", "EWA_Australia_zscore_60d", "BTI_BritishAmerican_ret_5d"], "is_new": true}, {"model_id": "new_h3_STRESS_LogisticRegression_N5_t6", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 3, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "BDX_Becton_Dickinson_ret_20d", "MS_MorganStanley_ret_1d", "NWL_Newell_ret_20d"], "is_new": true}, {"model_id": "new_h3_STRESS_LogisticRegression_N5_t7", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 3, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "DE_Deere_vol_20d", "CPB_CampbellSoup_ret_5d", "TXN_vol_20d"], "is_new": true}, {"model_id": "new_h3_STRESS_LogisticRegression_N8_t0", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 3, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "spx_abs_ret_max_5d", "BA_ret_1d", "HangSeng_HK_vol_20d", "SBUX_ret_5d", "AVB_AvalonBay_zscore_60d", "LUV_SouthwestAir_ret_5d"], "is_new": true}, {"model_id": "new_h3_STRESS_LogisticRegression_N8_t1", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 3, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "NOC_Northrop_ret_20d", "INTC_ret_5d", "MS_MorganStanley_ret_1d", "LLY_zscore_60d", "MSTR_Bitcoin3_ret_5d", "HD_ret_1d"], "is_new": true}, {"model_id": "new_h3_STRESS_LogisticRegression_N8_t2", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 3, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "MS_MorganStanley_ret_1d", "NVDA_vol_20d", "PFE_ret_1d", "CPB_CampbellSoup_ret_20d", "IYR_US_REIT2_zscore_60d", "CTAS_Cintas_vol_20d"], "is_new": true}, {"model_id": "new_h3_STRESS_LogisticRegression_N8_t3", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 3, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "SCHW_Schwab_ret_5d", "AMGN_Amgen_ret_1d", "CPB_CampbellSoup_vol_20d", "T10Y2Y_Spread_ret_5d", "spx_momentum_3d", "Industrial_Production_zscore_60d"], "is_new": true}, {"model_id": "new_h3_STRESS_LogisticRegression_N8_t4", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 3, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "SBUX_zscore_60d", "gjr_condvar_h1", "VOD_Vodafone_zscore_60d", "ES_Evergy_ret_1d", "NEE_NextEra_ret_20d", "EQR_Equity_ret_1d"], "is_new": true}, {"model_id": "new_h3_STRESS_LogisticRegression_N8_t5", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 3, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "PFE_ret_1d", "NVDA_vol_20d", "CCI_CrownCastle_vol_20d", "LMT_LockheedMartin_ret_1d", "AORD_AUS_zscore_60d", "EWQ_France_ret_20d"], "is_new": true}, {"model_id": "new_h3_STRESS_LogisticRegression_N8_t6", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 3, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "SPY_zscore_60d", "NOC_Northrop_ret_20d", "BTI_BritishAmerican_ret_20d", "SLB_Schlumberger_ret_1d", "Nikkei_Japan_zscore_60d", "vix_mean_abs_ret_5d"], "is_new": true}, {"model_id": "new_h3_STRESS_LogisticRegression_N8_t7", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 3, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWG_Germany_vol_20d", "HD_ret_1d", "DIS_vol_20d", "EMR_Emerson_ret_20d", "VRP_ma5", "XLB_Materials_zscore_60d"], "is_new": true}, {"model_id": "new_h3_STRESS_LogisticRegression_N10_t0", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 3, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "DE_Deere_ret_5d", "DAX_Germany_vol_20d", "3M_ret_5d", "BTI_BritishAmerican_ret_20d", "EWQ_France_zscore_60d", "EWL_Switzerland_zscore_60d", "US7Y_Rate_ret_20d", "TXN_vol_20d"], "is_new": true}, {"model_id": "new_h3_STRESS_LogisticRegression_N10_t1", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 3, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "HD_ret_1d", "3M_vol_20d", "AXP_Amex_vol_20d", "EWY_Korea_ret_20d", "SBUX_zscore_60d", "NVDA_vol_20d", "LLY_zscore_60d", "US1Y_Rate_ret_20d"], "is_new": true}, {"model_id": "new_h3_STRESS_LogisticRegression_N10_t2", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 3, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "AVB_AvalonBay_zscore_60d", "Industrial_Production_zscore_60d", "Nikkei_Japan_zscore_60d", "IYR_US_REIT2_zscore_60d", "3M_ret_5d", "DIS_vol_20d", "LMT_LockheedMartin_vol_20d", "PAYX_Paychex_zscore_60d"], "is_new": true}, {"model_id": "new_h3_STRESS_LogisticRegression_N10_t3", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 3, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "XLK_Tech_zscore_60d", "IWM_SmallCap_vol_20d", "BDX_Becton_Dickinson_ret_20d", "DHR_vol_20d", "PG_ret_20d", "XLB_Materials_zscore_60d", "US3M_Rate_vol_20d", "3M_vol_20d"], "is_new": true}, {"model_id": "new_h3_STRESS_LogisticRegression_N10_t4", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 3, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "XLB_Materials_zscore_60d", "BTI_BritishAmerican_ret_5d", "VVIX_ret_20d", "US5Y_Rate_ret_5d", "CTAS_Cintas_vol_20d", "SPY_zscore_60d", "EWA_Australia_zscore_60d", "T10Y2Y_Spread_ret_5d"], "is_new": true}, {"model_id": "new_h3_STRESS_LogisticRegression_N10_t5", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 3, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "DE_Deere_vol_20d", "NEE_NextEra_ret_20d", "VOD_Vodafone_zscore_60d", "NOC_Northrop_ret_20d", "CCI_CrownCastle_vol_20d", "NFCI_ret_5d", "EWA_Australia_zscore_60d", "MS_MorganStanley_ret_1d"], "is_new": true}, {"model_id": "new_h3_STRESS_LogisticRegression_N10_t6", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 3, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "PFE_ret_1d", "INTC_ret_5d", "ES_Evergy_ret_1d", "WTI_Oil_FRED_zscore_60d", "NFCI_ret_5d", "US3Y_Rate_ret_5d", "HangSeng_HK_ret_1d", "EQR_Equity_ret_1d"], "is_new": true}, {"model_id": "new_h3_STRESS_LogisticRegression_N10_t7", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 3, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "GE_ret_1d", "TM_Telephone_vol_20d", "gjr_condvar_h1", "XLV_Health_zscore_60d", "PG_ret_20d", "TXN_vol_20d", "CLX_Clorox_vol_20d", "vix_mean_abs_ret_5d"], "is_new": true}, {"model_id": "new_h3_STRESS_LogisticRegression_N12_t0", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 3, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EXC_Exelon_zscore_60d", "IWM_SmallCap_vol_20d", "EMR_Emerson_ret_20d", "PAYX_Paychex_ret_20d", "PPL_PPL_ret_1d", "ASX_Australia_vol_20d", "LOW_Lowes_ret_5d", "TGT_Target_zscore_60d", "Industrial_Production_zscore_60d", "AVB_AvalonBay_zscore_60d"], "is_new": true}, {"model_id": "new_h3_STRESS_LogisticRegression_N12_t1", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 3, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "HUM_Humana_ret_5d", "TGT_Target_zscore_60d", "CCI_CrownCastle_vol_20d", "TXN_vol_20d", "TM_Telephone_ret_1d", "spx_momentum_3d", "MSTR_Bitcoin3_ret_5d", "PAYX_Paychex_ret_20d", "SPY_zscore_60d", "VRP_ma5"], "is_new": true}, {"model_id": "new_h3_STRESS_LogisticRegression_N12_t2", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 3, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWM_Malaysia_ret_1d", "US3Y_Rate_ret_5d", "CTAS_Cintas_vol_20d", "US5Y_Rate_ret_5d", "EWG_Germany_ret_20d", "TED_Spread_vol_20d", "PG_ret_20d", "BLK_BlackRock_zscore_60d", "ORCL_zscore_60d", "HD_ret_1d"], "is_new": true}, {"model_id": "new_h3_STRESS_LogisticRegression_N12_t3", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 3, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "AMT_AmericanTower_ret_1d", "CPB_CampbellSoup_ret_5d", "EWQ_France_zscore_60d", "GILD_Gilead_ret_20d", "XOM_ret_1d", "spx_momentum_3d", "TM_Telephone_ret_1d", "IBEX_Spain_ret_20d", "EWJ_Japan_vol_20d", "US1Y_Rate_ret_20d"], "is_new": true}, {"model_id": "new_h3_STRESS_LogisticRegression_N12_t4", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 3, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "AMGN_Amgen_ret_1d", "HD_zscore_60d", "MS_MorganStanley_ret_1d", "EQIX_Equinix_ret_5d", "IBEX_Spain_ret_20d", "QQQ_vol_20d", "TGT_Target_zscore_60d", "ORCL_vol_20d", "EWC_Canada_zscore_60d", "LOW_Lowes_ret_20d"], "is_new": true}, {"model_id": "new_h3_STRESS_LogisticRegression_N12_t5", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 3, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "spx_vol_5d", "EMR_Emerson_ret_20d", "SBUX_ret_5d", "EWM_Malaysia_vol_20d", "EWM_Malaysia_zscore_60d", "AXP_Amex_vol_20d", "SJM_JM_Smucker_ret_5d", "US3Y_Rate_ret_5d", "AMD_ret_1d", "EWQ_France_zscore_60d"], "is_new": true}, {"model_id": "new_h3_STRESS_LogisticRegression_N12_t6", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 3, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "MS_MorganStanley_zscore_60d", "spx_abs_ret_max_5d", "SJM_JM_Smucker_ret_5d", "HD_ret_20d", "EWY_Korea_ret_20d", "EQIX_Equinix_ret_5d", "BTI_BritishAmerican_ret_20d", "AVB_AvalonBay_zscore_60d", "VVIX_ret_20d", "ITT_ITTInc_ret_5d"], "is_new": true}, {"model_id": "new_h3_STRESS_LogisticRegression_N12_t7", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 3, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "SBUX_vol_20d", "PFE_ret_1d", "vix_acceleration_1d", "EWA_Australia_ret_1d", "BDX_Becton_Dickinson_ret_20d", "PPL_PPL_ret_1d", "US3M_Rate_vol_20d", "3M_ret_5d", "US3Y_Rate_ret_5d", "XLY_Disc_vol_20d"], "is_new": true}, {"model_id": "new_h3_STRESS_LogisticRegression_N15_t0", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 3, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "NVDA_vol_20d", "BTI_BritishAmerican_ret_5d", "PLD_Prologis_ret_5d", "AMGN_Amgen_ret_1d", "VVIX_ret_20d", "US5Y_Rate_ret_5d", "MRK_Merck_zscore_60d", "Core_PCE_zscore_60d", "CMCSA_ret_1d", "AVB_AvalonBay_zscore_60d", "EWL_Switzerland_vol_20d", "LMT_LockheedMartin_ret_1d", "Michigan_Sentiment_ret_20d"], "is_new": true}, {"model_id": "new_h3_STRESS_LogisticRegression_N15_t1", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 3, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "MS_MorganStanley_zscore_60d", "DE_Deere_vol_20d", "BTI_BritishAmerican_ret_5d", "BDX_Becton_Dickinson_ret_20d", "XOM_ret_20d", "VVIX_ret_20d", "ASX_Australia_ret_5d", "DHR_vol_20d", "TED_Spread_vol_20d", "AMZN_ret_5d", "LLY_zscore_60d", "PCAR_PaccarInc_ret_5d", "VRP_ma5"], "is_new": true}, {"model_id": "new_h3_STRESS_LogisticRegression_N15_t2", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 3, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "AMZN_ret_5d", "BDX_Becton_Dickinson_ret_20d", "EXC_Exelon_ret_1d", "HangSeng_HK_ret_1d", "GILD_Gilead_ret_20d", "Brent_Oil_FRED_ret_20d", "LLY_zscore_60d", "Nikkei_Japan_vol_20d", "MSTR_Bitcoin3_ret_20d", "EWQ_France_ret_20d", "AVB_AvalonBay_zscore_60d", "MRK_Merck_zscore_60d", "PAYX_Paychex_vol_20d"], "is_new": true}, {"model_id": "new_h3_STRESS_LogisticRegression_N15_t3", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 3, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EFFR_ret_1d", "ES_Evergy_ret_1d", "AMT_AmericanTower_ret_1d", "EMR_Emerson_ret_20d", "Core_CPI_zscore_60d", "PAYX_Paychex_zscore_60d", "INTC_ret_5d", "TM_Telephone_vol_20d", "US3M_Rate_zscore_60d", "EQIX_Equinix_ret_5d", "US30Y_Rate_ret_20d", "EWJ_Japan_vol_20d", "US3M_Rate_vol_20d"], "is_new": true}, {"model_id": "new_h3_STRESS_LogisticRegression_N15_t4", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 3, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "PFE_ret_1d", "AORD_AUS_zscore_60d", "Core_PCE_zscore_60d", "EWM_Malaysia_vol_20d", "heston_var_ev_h5", "Retail_Sales_zscore_60d", "MS_MorganStanley_ret_5d", "vix_acceleration_1d", "PCAR_PaccarInc_ret_5d", "ES_Evergy_ret_1d", "3M_vol_20d", "vix_mean_abs_ret_5d", "EWL_Switzerland_vol_20d"], "is_new": true}, {"model_id": "new_h3_STRESS_LogisticRegression_N15_t5", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 3, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWM_Malaysia_vol_20d", "heston_var_ev_h7", "PLD_Prologis_ret_5d", "SBUX_ret_5d", "LUV_SouthwestAir_ret_5d", "NVDA_vol_20d", "SBUX_zscore_60d", "QQQ_vol_20d", "XLK_Tech_zscore_60d", "GE_ret_1d", "MSTR_Bitcoin3_ret_1d", "MS_MorganStanley_ret_1d", "TXN_vol_20d"], "is_new": true}, {"model_id": "new_h3_STRESS_LogisticRegression_N15_t6", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 3, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "Core_CPI_zscore_60d", "HUM_Humana_ret_5d", "SLB_Schlumberger_ret_1d", "HD_ret_20d", "EFFR_ret_1d", "NVDA_vol_20d", "hmm_p_stress", "US30Y_Rate_ret_20d", "XLB_Materials_zscore_60d", "SLB_Schlumberger_ret_5d", "T_ret_1d", "ASX_Australia_vol_20d", "SBUX_vol_20d"], "is_new": true}, {"model_id": "new_h3_STRESS_LogisticRegression_N15_t7", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 3, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "ORCL_vol_20d", "LLY_zscore_60d", "TED_Spread_zscore_60d", "Industrial_Production_zscore_60d", "ORCL_zscore_60d", "T_ret_1d", "MRK_Merck_zscore_60d", "MS_MorganStanley_ret_5d", "HangSeng_HK_ret_1d", "LUV_SouthwestAir_ret_5d", "CLX_Clorox_vol_20d", "US6M_Rate_ret_20d", "EQIX_Equinix_ret_5d"], "is_new": true}, {"model_id": "new_h3_STRESS_LogisticRegression_N20_t0", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 3, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "FedFunds_zscore_60d", "MSTR_Bitcoin3_ret_5d", "TM_Telephone_ret_1d", "DAX_Germany_zscore_60d", "spx_abs_ret_max_5d", "XLV_Health_zscore_60d", "vix_mean_abs_ret_5d", "US30Y_Rate_ret_20d", "LOW_Lowes_ret_5d", "CMCSA_ret_1d", "M_Macys_vol_20d", "LLY_zscore_60d", "DHR_vol_20d", "EQR_Equity_ret_1d", "vix_acceleration_1d", "T10Y2Y_Spread_ret_5d", "SBUX_zscore_60d", "EWH_HongKong_ret_5d"], "is_new": true}, {"model_id": "new_h3_STRESS_LogisticRegression_N20_t1", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 3, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "SBUX_ret_5d", "TED_Spread_zscore_60d", "MS_MorganStanley_ret_5d", "EWQ_France_zscore_60d", "EXC_Exelon_zscore_60d", "XOM_ret_20d", "BTI_BritishAmerican_ret_5d", "GILD_Gilead_ret_20d", "HD_ret_20d", "US30Y_Rate_ret_20d", "Retail_Sales_zscore_60d", "vix_acceleration_1d", "SBUX_vol_20d", "ORCL_zscore_60d", "HangSeng_HK_ret_1d", "PAYX_Paychex_ret_20d", "NEE_NextEra_ret_20d", "EWC_Canada_zscore_60d"], "is_new": true}, {"model_id": "new_h3_STRESS_LogisticRegression_N20_t2", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 3, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWA_Australia_zscore_60d", "BTI_BritishAmerican_ret_5d", "TED_Spread_vol_20d", "TM_Telephone_ret_1d", "ORCL_zscore_60d", "EQIX_Equinix_ret_5d", "SJM_JM_Smucker_ret_1d", "SBUX_vol_20d", "US3M_Rate_zscore_60d", "NVDA_vol_20d", "BDX_Becton_Dickinson_ret_20d", "NEE_NextEra_ret_20d", "vix_acceleration_1d", "TM_Telephone_vol_20d", "SLB_Schlumberger_ret_1d", "T10Y2Y_Spread_ret_5d", "vix_mean_abs_ret_5d", "MSTR_Bitcoin3_ret_20d"], "is_new": true}, {"model_id": "new_h3_STRESS_LogisticRegression_N20_t3", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 3, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "IYM_BasicMaterials_ret_20d", "AORD_AUS_zscore_60d", "vix_acceleration_1d", "IWM_SmallCap_vol_20d", "PPL_PPL_ret_1d", "TGT_Target_zscore_60d", "HUM_Humana_ret_5d", "Michigan_Sentiment_ret_20d", "EOG_EOGResources_vol_20d", "DHR_vol_20d", "BTI_BritishAmerican_ret_5d", "EWH_HongKong_ret_5d", "FedFunds_zscore_60d", "NEE_NextEra_ret_20d", "CMCSA_ret_1d", "INTC_ret_5d", "Core_CPI_zscore_60d", "LLY_zscore_60d"], "is_new": true}, {"model_id": "new_h3_STRESS_LogisticRegression_N20_t4", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 3, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "HUM_Humana_ret_5d", "Brent_Oil_FRED_ret_20d", "EWM_Malaysia_zscore_60d", "T10Y2Y_Spread_ret_5d", "DAX_Germany_zscore_60d", "XOM_ret_20d", "XLV_Health_zscore_60d", "SJM_JM_Smucker_ret_5d", "heston_var_ev_h5", "SBUX_ret_5d", "XOM_ret_1d", "AXP_Amex_vol_20d", "TGT_Target_zscore_60d", "Nikkei_Japan_zscore_60d", "spx_vol_5d", "SLB_Schlumberger_ret_5d", "Nikkei_Japan_vol_20d", "INTC_ret_1d"], "is_new": true}, {"model_id": "new_h3_STRESS_LogisticRegression_N20_t5", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 3, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "BA_ret_1d", "CPB_CampbellSoup_vol_20d", "SCHW_Schwab_ret_5d", "AMD_ret_5d", "LMT_LockheedMartin_ret_1d", "US3M_Rate_zscore_60d", "EWQ_France_ret_20d", "XLV_Health_zscore_60d", "NOC_Northrop_ret_20d", "PG_ret_20d", "ASX_Australia_vol_20d", "CPB_CampbellSoup_zscore_60d", "IYR_US_REIT2_zscore_60d", "MO_AltriaMG_ret_1d", "ENB_EnbridgeInc_ret_1d", "ES_Evergy_ret_1d", "MS_MorganStanley_ret_1d", "HangSeng_HK_vol_20d"], "is_new": true}, {"model_id": "new_h3_STRESS_LogisticRegression_N20_t6", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 3, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "MRK_Merck_zscore_60d", "IYR_US_REIT2_zscore_60d", "EFFR_ret_1d", "spx_momentum_3d", "WTI_Oil_FRED_zscore_60d", "LOW_Lowes_ret_5d", "SBUX_vol_20d", "US30Y_Rate_ret_20d", "PAYX_Paychex_zscore_60d", "EWC_Canada_zscore_60d", "INTC_ret_5d", "Core_CPI_zscore_60d", "MSTR_Bitcoin3_ret_20d", "TXN_vol_20d", "EWQ_France_zscore_60d", "DHR_ret_1d", "3M_vol_20d", "EQR_Equity_ret_1d"], "is_new": true}, {"model_id": "new_h3_STRESS_LogisticRegression_N20_t7", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 3, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWJ_Japan_vol_20d", "spx_abs_ret_max_5d", "CCI_CrownCastle_vol_20d", "SJM_JM_Smucker_ret_5d", "AXP_Amex_vol_20d", "BLK_BlackRock_zscore_60d", "DHR_vol_20d", "CI_Cigna_vol_20d", "HD_ret_1d", "WTI_Oil_FRED_zscore_60d", "SPY_zscore_60d", "BA_ret_1d", "ASX_Australia_ret_5d", "TGT_Target_zscore_60d", "EXC_Exelon_ret_1d", "PAYX_Paychex_ret_20d", "INTC_ret_1d", "IYR_US_REIT2_zscore_60d"], "is_new": true}, {"model_id": "new_h3_STRESS_LogisticRegression_N25_t0", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 3, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "HD_zscore_60d", "heston_var_ev_h7", "EWL_Switzerland_vol_20d", "DOW_Price_zscore_60d", "US30Y_Rate_ret_20d", "EQR_Equity_ret_1d", "EWM_Malaysia_vol_20d", "XLF_Fin_vol_20d", "BTI_BritishAmerican_ret_5d", "EWA_Australia_ret_1d", "DAX_Germany_zscore_60d", "BTI_BritishAmerican_ret_20d", "T_ret_1d", "EWQ_France_zscore_60d", "NOC_Northrop_ret_20d", "MS_MorganStanley_ret_1d", "US6M_Rate_ret_20d", "FedFunds_zscore_60d", "SPY_zscore_60d", "EWY_Korea_ret_20d", "LOW_Lowes_ret_20d", "VOD_Vodafone_zscore_60d", "SO_SouthernCo_ret_5d"], "is_new": true}, {"model_id": "new_h3_STRESS_LogisticRegression_N25_t1", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 3, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "WTI_Oil_FRED_zscore_60d", "SPY_zscore_60d", "SJM_JM_Smucker_ret_1d", "EWG_Germany_vol_20d", "DHR_ret_1d", "US5Y_Rate_ret_5d", "PCAR_PaccarInc_ret_5d", "INTC_ret_1d", "SLB_Schlumberger_ret_5d", "GE_ret_1d", "IYR_US_REIT2_zscore_60d", "Nikkei_Japan_vol_20d", "DHR_vol_20d", "QQQ_vol_20d", "CPB_CampbellSoup_zscore_60d", "EWA_Australia_ret_1d", "PLD_Prologis_ret_5d", "BA_ret_1d", "XOM_ret_1d", "EXC_Exelon_zscore_60d", "HangSeng_HK_ret_1d", "MS_MorganStanley_ret_5d", "TED_Spread_vol_20d"], "is_new": true}, {"model_id": "new_h3_STRESS_LogisticRegression_N25_t2", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 3, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "MSTR_Bitcoin3_ret_20d", "M_Macys_vol_20d", "AORD_AUS_zscore_60d", "US3Y_Rate_ret_5d", "HangSeng_HK_ret_1d", "PLD_Prologis_ret_5d", "TED_Spread_vol_20d", "EWC_Canada_zscore_60d", "BA_ret_1d", "NOC_Northrop_ret_20d", "DIS_vol_20d", "NWL_Newell_ret_20d", "HD_zscore_60d", "Michigan_Sentiment_ret_20d", "IYM_BasicMaterials_ret_20d", "SJM_JM_Smucker_ret_5d", "TGT_Target_zscore_60d", "US3M_Rate_vol_20d", "VRP_ma5", "US1Y_Rate_ret_20d", "SJM_JM_Smucker_ret_1d", "EQIX_Equinix_ret_5d", "GD_GeneralDynamics_zscore_60d"], "is_new": true}, {"model_id": "new_h3_STRESS_LogisticRegression_N25_t3", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 3, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "CI_Cigna_vol_20d", "DHR_vol_20d", "TED_Spread_zscore_60d", "HD_ret_1d", "DHR_ret_1d", "AVB_AvalonBay_zscore_60d", "US6M_Rate_ret_20d", "EOG_EOGResources_vol_20d", "PFE_ret_1d", "HangSeng_HK_vol_20d", "US7Y_Rate_ret_20d", "EWS_Singapore_ret_5d", "US3M_Rate_zscore_60d", "AXP_Amex_ret_20d", "VOD_Vodafone_zscore_60d", "SPY_zscore_60d", "CMCSA_ret_1d", "CCI_CrownCastle_vol_20d", "vix_acceleration_1d", "spx_abs_ret_max_5d", "ASX_Australia_vol_20d", "EQR_Equity_ret_1d", "Brent_Oil_FRED_ret_20d"], "is_new": true}, {"model_id": "new_h3_STRESS_LogisticRegression_N25_t4", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 3, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "CCI_CrownCastle_vol_20d", "EWH_HongKong_ret_5d", "MSTR_Bitcoin3_ret_5d", "SLB_Schlumberger_ret_5d", "HD_ret_1d", "ASX_Australia_ret_5d", "IWM_SmallCap_vol_20d", "HD_zscore_60d", "CPB_CampbellSoup_zscore_60d", "MS_MorganStanley_ret_1d", "IYM_BasicMaterials_ret_20d", "ENB_EnbridgeInc_ret_1d", "EWM_Malaysia_vol_20d", "SBUX_zscore_60d", "AMD_ret_1d", "3M_ret_5d", "TM_Telephone_ret_1d", "PAYX_Paychex_zscore_60d", "SJM_JM_Smucker_ret_1d", "QQQ_vol_20d", "MRK_Merck_zscore_60d", "Brent_Oil_FRED_ret_5d", "BA_ret_1d"], "is_new": true}, {"model_id": "new_h3_STRESS_LogisticRegression_N25_t5", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 3, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "Nikkei_Japan_zscore_60d", "AMZN_ret_5d", "QQQ_vol_20d", "EWQ_France_zscore_60d", "HD_ret_5d", "PG_ret_20d", "SLB_Schlumberger_ret_5d", "spx_abs_ret_max_5d", "spx_momentum_3d", "EMR_Emerson_ret_20d", "Nikkei_Japan_vol_20d", "PLD_Prologis_ret_5d", "EXC_Exelon_ret_1d", "TGT_Target_zscore_60d", "IBEX_Spain_ret_20d", "NWL_Newell_ret_20d", "GE_ret_1d", "EWL_Switzerland_zscore_60d", "Brent_Oil_FRED_ret_5d", "BLK_BlackRock_zscore_60d", "EWH_HongKong_ret_5d", "DHR_ret_1d", "EWL_Switzerland_vol_20d"], "is_new": true}, {"model_id": "new_h3_STRESS_LogisticRegression_N25_t6", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 3, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "PAYX_Paychex_vol_20d", "ORCL_vol_20d", "SBUX_ret_5d", "EWQ_France_ret_20d", "BDX_Becton_Dickinson_ret_20d", "LOW_Lowes_ret_20d", "T_ret_1d", "JNJ_ret_1d", "TM_Telephone_ret_1d", "EOG_EOGResources_vol_20d", "Brent_Oil_FRED_ret_5d", "XLY_Disc_vol_20d", "NWL_Newell_ret_20d", "HD_ret_1d", "heston_var_ev_h5", "XLB_Materials_zscore_60d", "NVDA_vol_20d", "TXN_vol_20d", "DOW_Price_zscore_60d", "DHR_ret_1d", "GE_ret_1d", "EWG_Germany_vol_20d", "CMCSA_ret_1d"], "is_new": true}, {"model_id": "new_h3_STRESS_LogisticRegression_N25_t7", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 3, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "LUV_SouthwestAir_ret_5d", "SJM_JM_Smucker_ret_5d", "vix_acceleration_1d", "XLV_Health_zscore_60d", "T10Y2Y_Spread_ret_5d", "EWC_Canada_zscore_60d", "spx_momentum_3d", "EOG_EOGResources_ret_5d", "EWG_Germany_vol_20d", "AMT_AmericanTower_ret_1d", "Retail_Sales_zscore_60d", "TXN_vol_20d", "Core_PCE_zscore_60d", "GD_GeneralDynamics_zscore_60d", "CTAS_Cintas_vol_20d", "DAX_Germany_zscore_60d", "IYM_BasicMaterials_ret_20d", "PLD_Prologis_ret_5d", "heston_ev_h3", "NFCI_ret_5d", "PAYX_Paychex_zscore_60d", "BDX_Becton_Dickinson_ret_20d", "PFE_ret_1d"], "is_new": true}, {"model_id": "new_h3_STRESS_LogisticRegression_N30_t0", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 3, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "JNJ_ret_1d", "HangSeng_HK_ret_1d", "PPL_PPL_ret_1d", "IWM_SmallCap_vol_20d", "spx_abs_ret_max_5d", "XLB_Materials_zscore_60d", "ORCL_zscore_60d", "BTI_BritishAmerican_ret_5d", "CPB_CampbellSoup_vol_20d", "EWM_Malaysia_zscore_60d", "CTAS_Cintas_vol_20d", "MRK_Merck_zscore_60d", "NOC_Northrop_ret_20d", "TED_Spread_vol_20d", "SBUX_vol_20d", "EWH_HongKong_ret_5d", "Core_CPI_zscore_60d", "AVB_AvalonBay_zscore_60d", "DE_Deere_ret_5d", "vix_acceleration_1d", "EFFR_vol_20d", "MS_MorganStanley_ret_1d", "HUM_Humana_ret_5d", "INTC_ret_5d", "Nikkei_Japan_vol_20d", "heston_ev_h3", "HD_ret_1d", "US5Y_Rate_ret_5d"], "is_new": true}, {"model_id": "new_h3_STRESS_LogisticRegression_N30_t1", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 3, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "CPB_CampbellSoup_ret_5d", "AMD_ret_5d", "US3Y_Rate_ret_5d", "vix_acceleration_1d", "EWA_Australia_zscore_60d", "LLY_zscore_60d", "US1Y_Rate_ret_5d", "hmm_p_stress", "SJM_JM_Smucker_ret_5d", "QQQ_vol_20d", "DE_Deere_vol_20d", "MSTR_Bitcoin3_ret_1d", "EWA_Australia_ret_1d", "ES_Evergy_ret_1d", "heston_ev_h3", "LMT_LockheedMartin_ret_1d", "ASX_Australia_ret_5d", "XLY_Disc_vol_20d", "SBUX_zscore_60d", "Brent_Oil_FRED_ret_20d", "EFFR_ret_1d", "MS_MorganStanley_ret_1d", "HangSeng_HK_vol_20d", "XLB_Materials_zscore_60d", "CTAS_Cintas_vol_20d", "3M_vol_20d", "EOG_EOGResources_vol_20d", "EWQ_France_ret_20d"], "is_new": true}, {"model_id": "new_h3_STRESS_LogisticRegression_N30_t2", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 3, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "SBUX_zscore_60d", "EWY_Korea_ret_20d", "ORCL_vol_20d", "NOC_Northrop_ret_20d", "heston_var_ev_h3", "HangSeng_HK_vol_20d", "US3M_Rate_zscore_60d", "SCHW_Schwab_ret_5d", "AVB_AvalonBay_zscore_60d", "MSTR_Bitcoin3_ret_1d", "XOM_ret_1d", "INTC_ret_1d", "US6M_Rate_ret_20d", "XLY_Disc_vol_20d", "3M_ret_5d", "Core_CPI_zscore_60d", "US3M_Rate_vol_20d", "Michigan_Sentiment_ret_20d", "GD_GeneralDynamics_zscore_60d", "EWL_Switzerland_zscore_60d", "EWQ_France_zscore_60d", "LLY_zscore_60d", "NFCI_ret_5d", "DHR_vol_20d", "EWA_Australia_ret_1d", "IYR_US_REIT2_zscore_60d", "AMZN_ret_5d", "EOG_EOGResources_vol_20d"], "is_new": true}, {"model_id": "new_h3_STRESS_LogisticRegression_N30_t3", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 3, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "3M_vol_20d", "AXP_Amex_ret_20d", "vix_mean_abs_ret_5d", "AVB_AvalonBay_zscore_60d", "TM_Telephone_ret_1d", "DHR_ret_1d", "EQIX_Equinix_ret_5d", "DAX_Germany_zscore_60d", "BDX_Becton_Dickinson_ret_20d", "LOW_Lowes_ret_20d", "EWQ_France_ret_20d", "DE_Deere_ret_5d", "ENB_EnbridgeInc_ret_1d", "spx_momentum_3d", "TM_Telephone_vol_20d", "VOD_Vodafone_zscore_60d", "gjr_condvar_h1", "US5Y_Rate_ret_5d", "heston_ev_h3", "MSTR_Bitcoin3_ret_5d", "AXP_Amex_vol_20d", "TED_Spread_zscore_60d", "US3Y_Rate_ret_5d", "XLK_Tech_zscore_60d", "AORD_AUS_zscore_60d", "HD_zscore_60d", "NWL_Newell_ret_20d", "VVIX_ret_20d"], "is_new": true}, {"model_id": "new_h3_STRESS_LogisticRegression_N30_t4", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 3, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "CMCSA_ret_1d", "HD_ret_5d", "PPL_PPL_ret_1d", "TM_Telephone_ret_1d", "EQR_Equity_ret_1d", "FedFunds_zscore_60d", "PLD_Prologis_ret_5d", "LMT_LockheedMartin_vol_20d", "US3M_Rate_vol_20d", "XLV_Health_zscore_60d", "CPB_CampbellSoup_ret_5d", "DAX_Germany_vol_20d", "SBUX_vol_20d", "EWA_Australia_zscore_60d", "US3M_Rate_zscore_60d", "AXP_Amex_ret_20d", "MS_MorganStanley_ret_5d", "NVDA_vol_20d", "ITT_ITTInc_ret_5d", "BDX_Becton_Dickinson_ret_20d", "Michigan_Sentiment_ret_20d", "SO_SouthernCo_ret_5d", "TED_Spread_zscore_60d", "SJM_JM_Smucker_ret_5d", "LUV_SouthwestAir_ret_5d", "PG_ret_20d", "Nikkei_Japan_vol_20d", "ASX_Australia_vol_20d"], "is_new": true}, {"model_id": "new_h3_STRESS_LogisticRegression_N30_t5", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 3, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "NWL_Newell_ret_20d", "AMT_AmericanTower_ret_1d", "EWQ_France_zscore_60d", "LMT_LockheedMartin_ret_1d", "BA_ret_1d", "EFFR_ret_1d", "HangSeng_HK_vol_20d", "JNJ_ret_1d", "AMD_ret_5d", "MS_MorganStanley_ret_5d", "EWG_Germany_ret_20d", "DAX_Germany_zscore_60d", "PG_ret_20d", "BLK_BlackRock_zscore_60d", "spx_momentum_3d", "hmm_p_stress", "GILD_Gilead_ret_20d", "CLX_Clorox_vol_20d", "Nikkei_Japan_vol_20d", "EXC_Exelon_ret_1d", "INTC_ret_1d", "Nikkei_Japan_zscore_60d", "HangSeng_HK_ret_1d", "MSTR_Bitcoin3_ret_20d", "TM_Telephone_ret_1d", "NOC_Northrop_ret_20d", "MSTR_Bitcoin3_ret_1d", "PLD_Prologis_ret_5d"], "is_new": true}, {"model_id": "new_h3_STRESS_LogisticRegression_N30_t6", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 3, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "HD_ret_1d", "ENB_EnbridgeInc_ret_1d", "SLB_Schlumberger_ret_1d", "HangSeng_HK_vol_20d", "AMD_ret_1d", "AMGN_Amgen_ret_1d", "3M_vol_20d", "AMD_ret_5d", "CTAS_Cintas_vol_20d", "EWJ_Japan_vol_20d", "PAYX_Paychex_ret_20d", "FedFunds_zscore_60d", "EWA_Australia_ret_1d", "PAYX_Paychex_vol_20d", "SLB_Schlumberger_ret_5d", "GE_ret_1d", "SBUX_zscore_60d", "CPB_CampbellSoup_ret_20d", "hmm_p_stress", "QQQ_vol_20d", "EWQ_France_zscore_60d", "spx_vol_5d", "EQIX_Equinix_ret_5d", "XOM_ret_1d", "EQR_Equity_ret_1d", "DIS_vol_20d", "IWM_SmallCap_vol_20d", "TM_Telephone_vol_20d"], "is_new": true}, {"model_id": "new_h3_STRESS_LogisticRegression_N30_t7", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 3, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "Nikkei_Japan_vol_20d", "ENB_EnbridgeInc_ret_1d", "heston_var_ev_h5", "DHR_vol_20d", "EWH_HongKong_ret_5d", "EWQ_France_ret_20d", "GD_GeneralDynamics_zscore_60d", "IYM_BasicMaterials_ret_20d", "AXP_Amex_ret_20d", "Core_CPI_zscore_60d", "JNJ_ret_1d", "GILD_Gilead_ret_20d", "Core_PCE_zscore_60d", "PPL_PPL_ret_1d", "SCHW_Schwab_ret_5d", "Michigan_Sentiment_ret_20d", "MO_AltriaMG_ret_1d", "PG_ret_20d", "TM_Telephone_ret_1d", "NVDA_vol_20d", "ORCL_vol_20d", "CCI_CrownCastle_vol_20d", "AVB_AvalonBay_zscore_60d", "AXP_Amex_vol_20d", "ASX_Australia_vol_20d", "AMGN_Amgen_ret_1d", "LMT_LockheedMartin_ret_1d", "CLX_Clorox_vol_20d"], "is_new": true}, {"model_id": "new_h3_GLOBAL_XGBoost_N5_t0", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 3, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "SJM_JM_Smucker_ret_1d", "XLF_Fin_vol_20d", "EWQ_France_zscore_60d"], "is_new": true}, {"model_id": "new_h3_GLOBAL_XGBoost_N5_t1", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 3, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "SLB_Schlumberger_ret_5d", "vix_acceleration_1d", "US5Y_Rate_ret_5d"], "is_new": true}, {"model_id": "new_h3_GLOBAL_XGBoost_N5_t2", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 3, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "MS_MorganStanley_zscore_60d", "VOD_Vodafone_zscore_60d", "heston_var_ev_h5"], "is_new": true}, {"model_id": "new_h3_GLOBAL_XGBoost_N5_t3", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 3, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "INTC_ret_5d", "LLY_zscore_60d", "heston_ev_h3"], "is_new": true}, {"model_id": "new_h3_GLOBAL_XGBoost_N5_t4", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 3, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "MO_AltriaMG_ret_1d", "SJM_JM_Smucker_ret_1d", "Industrial_Production_zscore_60d"], "is_new": true}, {"model_id": "new_h3_GLOBAL_XGBoost_N5_t5", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 3, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "CTAS_Cintas_vol_20d", "SLB_Schlumberger_ret_5d", "EWM_Malaysia_ret_1d"], "is_new": true}, {"model_id": "new_h3_GLOBAL_XGBoost_N5_t6", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 3, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "BA_ret_1d", "vix_acceleration_1d", "TM_Telephone_ret_1d"], "is_new": true}, {"model_id": "new_h3_GLOBAL_XGBoost_N5_t7", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 3, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "TM_Telephone_ret_1d", "heston_var_ev_h3", "INTC_ret_5d"], "is_new": true}, {"model_id": "new_h3_GLOBAL_XGBoost_N8_t0", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 3, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWS_Singapore_ret_5d", "SLB_Schlumberger_ret_1d", "LUV_SouthwestAir_ret_5d", "HangSeng_HK_ret_1d", "CPB_CampbellSoup_zscore_60d", "T_ret_1d"], "is_new": true}, {"model_id": "new_h3_GLOBAL_XGBoost_N8_t1", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 3, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "Michigan_Sentiment_ret_20d", "HD_ret_1d", "ITT_ITTInc_ret_5d", "PCAR_PaccarInc_ret_5d", "AXP_Amex_ret_20d", "BTI_BritishAmerican_ret_20d"], "is_new": true}, {"model_id": "new_h3_GLOBAL_XGBoost_N8_t2", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 3, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "SBUX_vol_20d", "spx_vol_5d", "ASX_Australia_vol_20d", "BTI_BritishAmerican_ret_5d", "AXP_Amex_vol_20d", "NWL_Newell_ret_20d"], "is_new": true}, {"model_id": "new_h3_GLOBAL_XGBoost_N8_t3", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 3, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "LLY_zscore_60d", "HUM_Humana_ret_5d", "US3M_Rate_vol_20d", "SBUX_vol_20d", "CI_Cigna_vol_20d", "ES_Evergy_ret_1d"], "is_new": true}, {"model_id": "new_h3_GLOBAL_XGBoost_N8_t4", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 3, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "TGT_Target_zscore_60d", "EFFR_ret_1d", "MS_MorganStanley_ret_1d", "ENB_EnbridgeInc_ret_1d", "heston_var_ev_h7", "SBUX_ret_5d"], "is_new": true}, {"model_id": "new_h3_GLOBAL_XGBoost_N8_t5", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 3, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "ORCL_vol_20d", "AMT_AmericanTower_ret_1d", "HangSeng_HK_ret_1d", "Core_CPI_zscore_60d", "SLB_Schlumberger_ret_1d", "XLV_Health_zscore_60d"], "is_new": true}, {"model_id": "new_h3_GLOBAL_XGBoost_N8_t6", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 3, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "Michigan_Sentiment_ret_20d", "CPB_CampbellSoup_ret_5d", "CI_Cigna_vol_20d", "US1Y_Rate_ret_5d", "ES_Evergy_ret_1d", "T_ret_1d"], "is_new": true}, {"model_id": "new_h3_GLOBAL_XGBoost_N8_t7", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 3, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "SBUX_ret_5d", "INTC_ret_1d", "SJM_JM_Smucker_ret_5d", "Nikkei_Japan_zscore_60d", "HangSeng_HK_ret_5d", "EWL_Switzerland_zscore_60d"], "is_new": true}, {"model_id": "new_h3_GLOBAL_XGBoost_N10_t0", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 3, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "ITT_ITTInc_ret_5d", "EWM_Malaysia_ret_1d", "EWC_Canada_zscore_60d", "MSTR_Bitcoin3_ret_1d", "CTAS_Cintas_vol_20d", "FedFunds_zscore_60d", "US6M_Rate_ret_20d", "DE_Deere_vol_20d"], "is_new": true}, {"model_id": "new_h3_GLOBAL_XGBoost_N10_t1", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 3, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "Core_PCE_zscore_60d", "spx_momentum_3d", "QQQ_vol_20d", "INTC_ret_5d", "heston_var_ev_h7", "MS_MorganStanley_zscore_60d", "TXN_vol_20d", "Nikkei_Japan_zscore_60d"], "is_new": true}, {"model_id": "new_h3_GLOBAL_XGBoost_N10_t2", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 3, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "VRP_ma5", "MSTR_Bitcoin3_ret_20d", "HD_ret_1d", "vix_acceleration_1d", "EOG_EOGResources_ret_5d", "EOG_EOGResources_vol_20d", "AORD_AUS_zscore_60d", "NOC_Northrop_ret_20d"], "is_new": true}, {"model_id": "new_h3_GLOBAL_XGBoost_N10_t3", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 3, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "TED_Spread_zscore_60d", "EWQ_France_zscore_60d", "BA_ret_1d", "AMD_ret_1d", "EWC_Canada_zscore_60d", "LMT_LockheedMartin_ret_1d", "TGT_Target_zscore_60d", "BLK_BlackRock_zscore_60d"], "is_new": true}, {"model_id": "new_h3_GLOBAL_XGBoost_N10_t4", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 3, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "XOM_ret_20d", "EWS_Singapore_ret_5d", "EXC_Exelon_ret_1d", "HD_zscore_60d", "XLV_Health_zscore_60d", "CPB_CampbellSoup_ret_20d", "AVB_AvalonBay_zscore_60d", "SLB_Schlumberger_ret_1d"], "is_new": true}, {"model_id": "new_h3_GLOBAL_XGBoost_N10_t5", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 3, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "US1Y_Rate_ret_5d", "TXN_vol_20d", "EWQ_France_zscore_60d", "CI_Cigna_vol_20d", "XLF_Fin_vol_20d", "ES_Evergy_ret_1d", "ORCL_zscore_60d", "EMR_Emerson_ret_20d"], "is_new": true}, {"model_id": "new_h3_GLOBAL_XGBoost_N10_t6", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 3, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "AXP_Amex_vol_20d", "SJM_JM_Smucker_ret_1d", "US7Y_Rate_ret_20d", "XOM_ret_20d", "EWA_Australia_ret_1d", "DHR_ret_1d", "SJM_JM_Smucker_ret_5d", "CCI_CrownCastle_vol_20d"], "is_new": true}, {"model_id": "new_h3_GLOBAL_XGBoost_N10_t7", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 3, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "CLX_Clorox_vol_20d", "SBUX_zscore_60d", "PAYX_Paychex_zscore_60d", "vix_acceleration_1d", "CPB_CampbellSoup_ret_20d", "Core_PCE_zscore_60d", "3M_vol_20d", "EWM_Malaysia_ret_1d"], "is_new": true}, {"model_id": "new_h3_GLOBAL_XGBoost_N12_t0", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 3, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "SBUX_zscore_60d", "EWL_Switzerland_zscore_60d", "ORCL_vol_20d", "BTI_BritishAmerican_ret_20d", "SJM_JM_Smucker_ret_1d", "IBEX_Spain_ret_20d", "INTC_ret_5d", "PCAR_PaccarInc_ret_5d", "EXC_Exelon_zscore_60d", "CPB_CampbellSoup_zscore_60d"], "is_new": true}, {"model_id": "new_h3_GLOBAL_XGBoost_N12_t1", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 3, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "AMGN_Amgen_ret_1d", "BLK_BlackRock_zscore_60d", "AORD_AUS_zscore_60d", "AMD_ret_5d", "CMCSA_ret_1d", "EWM_Malaysia_vol_20d", "EWQ_France_zscore_60d", "heston_var_ev_h3", "EOG_EOGResources_ret_5d", "Core_PCE_zscore_60d"], "is_new": true}, {"model_id": "new_h3_GLOBAL_XGBoost_N12_t2", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 3, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "CCI_CrownCastle_vol_20d", "CMCSA_ret_1d", "EWY_Korea_ret_20d", "heston_var_ev_h5", "US3Y_Rate_ret_5d", "EMR_Emerson_ret_20d", "CPB_CampbellSoup_zscore_60d", "LMT_LockheedMartin_vol_20d", "spx_vol_5d", "heston_ev_h3"], "is_new": true}, {"model_id": "new_h3_GLOBAL_XGBoost_N12_t3", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 3, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "SBUX_zscore_60d", "DIS_vol_20d", "GD_GeneralDynamics_zscore_60d", "HD_ret_5d", "EWY_Korea_ret_20d", "heston_var_ev_h7", "PAYX_Paychex_vol_20d", "NVDA_vol_20d", "ITT_ITTInc_ret_5d", "MSTR_Bitcoin3_ret_1d"], "is_new": true}, {"model_id": "new_h3_GLOBAL_XGBoost_N12_t4", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 3, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "MRK_Merck_zscore_60d", "heston_ev_h3", "TXN_vol_20d", "EWC_Canada_zscore_60d", "EWG_Germany_vol_20d", "MS_MorganStanley_zscore_60d", "IYM_BasicMaterials_ret_20d", "XLY_Disc_vol_20d", "PAYX_Paychex_ret_20d", "IBEX_Spain_ret_20d"], "is_new": true}, {"model_id": "new_h3_GLOBAL_XGBoost_N12_t5", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 3, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "SO_SouthernCo_ret_5d", "PG_ret_20d", "Core_PCE_zscore_60d", "XLB_Materials_zscore_60d", "ES_Evergy_ret_1d", "XLF_Fin_vol_20d", "AMD_ret_5d", "SLB_Schlumberger_ret_5d", "EWM_Malaysia_ret_1d", "T10Y2Y_Spread_ret_5d"], "is_new": true}, {"model_id": "new_h3_GLOBAL_XGBoost_N12_t6", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 3, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "DAX_Germany_vol_20d", "US3M_Rate_zscore_60d", "PPL_PPL_ret_1d", "TGT_Target_zscore_60d", "heston_ev_h3", "US6M_Rate_ret_20d", "MSTR_Bitcoin3_ret_1d", "TXN_vol_20d", "Brent_Oil_FRED_ret_20d", "DE_Deere_vol_20d"], "is_new": true}, {"model_id": "new_h3_GLOBAL_XGBoost_N12_t7", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 3, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "MSTR_Bitcoin3_ret_5d", "US3Y_Rate_ret_5d", "US3M_Rate_zscore_60d", "EFFR_vol_20d", "PFE_ret_1d", "PAYX_Paychex_zscore_60d", "AMZN_ret_5d", "SJM_JM_Smucker_ret_5d", "heston_ev_h3", "NWL_Newell_ret_20d"], "is_new": true}, {"model_id": "new_h3_GLOBAL_XGBoost_N15_t0", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 3, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "HUM_Humana_ret_5d", "TM_Telephone_ret_1d", "NWL_Newell_ret_20d", "MRK_Merck_zscore_60d", "VVIX_ret_20d", "PAYX_Paychex_zscore_60d", "PAYX_Paychex_ret_20d", "spx_vol_5d", "INTC_ret_5d", "WTI_Oil_FRED_zscore_60d", "PG_ret_20d", "SBUX_zscore_60d", "ORCL_vol_20d"], "is_new": true}, {"model_id": "new_h3_GLOBAL_XGBoost_N15_t1", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 3, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "PG_ret_20d", "INTC_ret_5d", "ASX_Australia_ret_5d", "NFCI_ret_5d", "TED_Spread_vol_20d", "DE_Deere_vol_20d", "EWA_Australia_zscore_60d", "spx_abs_ret_max_5d", "EWY_Korea_ret_20d", "QQQ_vol_20d", "EWQ_France_ret_20d", "EWM_Malaysia_ret_1d", "US3M_Rate_vol_20d"], "is_new": true}, {"model_id": "new_h3_GLOBAL_XGBoost_N15_t2", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 3, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "CMCSA_ret_1d", "vix_acceleration_1d", "EOG_EOGResources_vol_20d", "ORCL_vol_20d", "SJM_JM_Smucker_ret_1d", "US30Y_Rate_ret_20d", "SJM_JM_Smucker_ret_5d", "Michigan_Sentiment_ret_20d", "Industrial_Production_zscore_60d", "PAYX_Paychex_vol_20d", "BA_ret_1d", "EWS_Singapore_ret_5d", "heston_var_ev_h7"], "is_new": true}, {"model_id": "new_h3_GLOBAL_XGBoost_N15_t3", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 3, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "3M_ret_5d", "LOW_Lowes_ret_5d", "EFFR_ret_1d", "CCI_CrownCastle_vol_20d", "PFE_ret_1d", "EWH_HongKong_ret_5d", "DE_Deere_vol_20d", "BTI_BritishAmerican_ret_5d", "EWY_Korea_ret_20d", "EWL_Switzerland_vol_20d", "heston_var_ev_h3", "IYR_US_REIT2_zscore_60d", "SO_SouthernCo_ret_5d"], "is_new": true}, {"model_id": "new_h3_GLOBAL_XGBoost_N15_t4", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 3, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EFFR_vol_20d", "VOD_Vodafone_zscore_60d", "spx_momentum_3d", "LLY_zscore_60d", "IYR_US_REIT2_zscore_60d", "SJM_JM_Smucker_ret_1d", "MS_MorganStanley_zscore_60d", "XLY_Disc_vol_20d", "EWL_Switzerland_vol_20d", "BDX_Becton_Dickinson_ret_20d", "SO_SouthernCo_ret_5d", "T10Y2Y_Spread_ret_5d", "TM_Telephone_ret_1d"], "is_new": true}, {"model_id": "new_h3_GLOBAL_XGBoost_N15_t5", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 3, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "WTI_Oil_FRED_zscore_60d", "US7Y_Rate_ret_20d", "BTI_BritishAmerican_ret_5d", "AMD_ret_1d", "NOC_Northrop_ret_20d", "DAX_Germany_zscore_60d", "EWA_Australia_ret_1d", "spx_vol_5d", "hmm_p_stress", "CI_Cigna_vol_20d", "LLY_zscore_60d", "HD_ret_20d", "EWM_Malaysia_ret_1d"], "is_new": true}, {"model_id": "new_h3_GLOBAL_XGBoost_N15_t6", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 3, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "IBEX_Spain_ret_20d", "PCAR_PaccarInc_ret_5d", "3M_ret_5d", "heston_var_ev_h3", "TXN_vol_20d", "HD_ret_5d", "TM_Telephone_vol_20d", "MS_MorganStanley_zscore_60d", "SBUX_vol_20d", "DAX_Germany_vol_20d", "CTAS_Cintas_vol_20d", "ORCL_zscore_60d", "EFFR_vol_20d"], "is_new": true}, {"model_id": "new_h3_GLOBAL_XGBoost_N15_t7", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 3, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EXC_Exelon_ret_1d", "EWG_Germany_ret_20d", "NWL_Newell_ret_20d", "MRK_Merck_zscore_60d", "AXP_Amex_vol_20d", "XLV_Health_zscore_60d", "EWY_Korea_zscore_60d", "CPB_CampbellSoup_vol_20d", "NOC_Northrop_ret_20d", "DHR_vol_20d", "gjr_condvar_h1", "MS_MorganStanley_ret_5d", "PCAR_PaccarInc_ret_5d"], "is_new": true}, {"model_id": "new_h3_GLOBAL_XGBoost_N20_t0", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 3, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EMR_Emerson_ret_20d", "US30Y_Rate_ret_20d", "vix_acceleration_1d", "SCHW_Schwab_ret_5d", "BDX_Becton_Dickinson_ret_20d", "CI_Cigna_vol_20d", "EWY_Korea_ret_20d", "DHR_vol_20d", "CPB_CampbellSoup_ret_5d", "MRK_Merck_zscore_60d", "MS_MorganStanley_zscore_60d", "NWL_Newell_ret_20d", "PAYX_Paychex_vol_20d", "BLK_BlackRock_zscore_60d", "BTI_BritishAmerican_ret_20d", "XLY_Disc_vol_20d", "Brent_Oil_FRED_ret_5d", "SLB_Schlumberger_ret_5d"], "is_new": true}, {"model_id": "new_h3_GLOBAL_XGBoost_N20_t1", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 3, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "3M_vol_20d", "GD_GeneralDynamics_zscore_60d", "IYR_US_REIT2_zscore_60d", "XLK_Tech_zscore_60d", "US3Y_Rate_ret_5d", "WTI_Oil_FRED_zscore_60d", "EFFR_vol_20d", "AXP_Amex_ret_20d", "QQQ_vol_20d", "CCI_CrownCastle_vol_20d", "SPY_zscore_60d", "XLF_Fin_vol_20d", "T_ret_1d", "vix_mean_abs_ret_5d", "AMT_AmericanTower_ret_1d", "DOW_Price_zscore_60d", "EOG_EOGResources_vol_20d", "EWS_Singapore_ret_5d"], "is_new": true}, {"model_id": "new_h3_GLOBAL_XGBoost_N20_t2", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 3, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "ASX_Australia_ret_5d", "EWA_Australia_ret_1d", "IBEX_Spain_ret_20d", "MSTR_Bitcoin3_ret_5d", "CPB_CampbellSoup_zscore_60d", "heston_ev_h3", "CMCSA_ret_1d", "QQQ_vol_20d", "EWQ_France_ret_20d", "HUM_Humana_ret_5d", "SBUX_ret_5d", "EXC_Exelon_ret_1d", "CLX_Clorox_vol_20d", "HD_ret_1d", "NEE_NextEra_ret_20d", "LLY_zscore_60d", "LUV_SouthwestAir_ret_5d", "HangSeng_HK_ret_1d"], "is_new": true}, {"model_id": "new_h3_GLOBAL_XGBoost_N20_t3", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 3, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWG_Germany_vol_20d", "JNJ_ret_1d", "LUV_SouthwestAir_ret_5d", "VOD_Vodafone_zscore_60d", "Brent_Oil_FRED_ret_5d", "NOC_Northrop_ret_20d", "HangSeng_HK_ret_5d", "XLY_Disc_vol_20d", "CLX_Clorox_vol_20d", "LMT_LockheedMartin_ret_1d", "spx_abs_ret_max_5d", "NFCI_ret_5d", "Michigan_Sentiment_ret_20d", "EWA_Australia_zscore_60d", "AMZN_ret_5d", "EWC_Canada_zscore_60d", "MSTR_Bitcoin3_ret_20d", "EWG_Germany_ret_20d"], "is_new": true}, {"model_id": "new_h3_GLOBAL_XGBoost_N20_t4", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 3, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "IYR_US_REIT2_zscore_60d", "AMGN_Amgen_ret_1d", "BA_ret_1d", "PAYX_Paychex_vol_20d", "HUM_Humana_ret_5d", "FedFunds_zscore_60d", "EWH_HongKong_ret_5d", "IWM_SmallCap_vol_20d", "EQIX_Equinix_ret_5d", "INTC_ret_5d", "hmm_p_stress", "3M_ret_5d", "BTI_BritishAmerican_ret_20d", "US3M_Rate_zscore_60d", "CLX_Clorox_vol_20d", "NFCI_ret_5d", "LUV_SouthwestAir_ret_5d", "EWM_Malaysia_zscore_60d"], "is_new": true}, {"model_id": "new_h3_GLOBAL_XGBoost_N20_t5", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 3, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "MS_MorganStanley_ret_5d", "hmm_p_stress", "ORCL_vol_20d", "XLV_Health_zscore_60d", "Nikkei_Japan_vol_20d", "XOM_ret_20d", "Retail_Sales_zscore_60d", "SBUX_vol_20d", "SLB_Schlumberger_ret_1d", "CPB_CampbellSoup_ret_5d", "EWH_HongKong_ret_5d", "FedFunds_zscore_60d", "BTI_BritishAmerican_ret_20d", "IBEX_Spain_ret_20d", "JNJ_ret_1d", "DOW_Price_zscore_60d", "GILD_Gilead_ret_20d", "T_ret_1d"], "is_new": true}, {"model_id": "new_h3_GLOBAL_XGBoost_N20_t6", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 3, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWL_Switzerland_vol_20d", "LLY_zscore_60d", "AMZN_ret_5d", "US3Y_Rate_ret_5d", "AMGN_Amgen_ret_1d", "vix_mean_abs_ret_5d", "FedFunds_zscore_60d", "EWC_Canada_zscore_60d", "LMT_LockheedMartin_ret_1d", "Brent_Oil_FRED_ret_5d", "T10Y2Y_Spread_ret_5d", "GILD_Gilead_ret_20d", "AVB_AvalonBay_zscore_60d", "MSTR_Bitcoin3_ret_1d", "EWH_HongKong_ret_5d", "WTI_Oil_FRED_zscore_60d", "EFFR_ret_1d", "EXC_Exelon_ret_1d"], "is_new": true}, {"model_id": "new_h3_GLOBAL_XGBoost_N20_t7", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 3, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWG_Germany_vol_20d", "DIS_vol_20d", "PCAR_PaccarInc_ret_5d", "CI_Cigna_vol_20d", "AXP_Amex_vol_20d", "ASX_Australia_ret_5d", "HangSeng_HK_ret_1d", "XLY_Disc_vol_20d", "spx_abs_ret_max_5d", "US7Y_Rate_ret_20d", "CPB_CampbellSoup_vol_20d", "EOG_EOGResources_vol_20d", "Nikkei_Japan_vol_20d", "HD_ret_1d", "EWQ_France_ret_20d", "SPY_zscore_60d", "FedFunds_zscore_60d", "PPL_PPL_ret_1d"], "is_new": true}, {"model_id": "new_h3_GLOBAL_XGBoost_N25_t0", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 3, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "heston_var_ev_h7", "AMGN_Amgen_ret_1d", "LOW_Lowes_ret_5d", "AXP_Amex_vol_20d", "XLF_Fin_vol_20d", "CPB_CampbellSoup_ret_20d", "MSTR_Bitcoin3_ret_5d", "EWJ_Japan_vol_20d", "EWQ_France_zscore_60d", "US3M_Rate_zscore_60d", "DOW_Price_zscore_60d", "EWS_Singapore_ret_5d", "MS_MorganStanley_ret_5d", "DAX_Germany_zscore_60d", "INTC_ret_5d", "US7Y_Rate_ret_20d", "EWL_Switzerland_zscore_60d", "DE_Deere_vol_20d", "US1Y_Rate_ret_20d", "Nikkei_Japan_vol_20d", "spx_vol_5d", "SBUX_zscore_60d", "Core_PCE_zscore_60d"], "is_new": true}, {"model_id": "new_h3_GLOBAL_XGBoost_N25_t1", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 3, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "DHR_ret_1d", "SLB_Schlumberger_ret_5d", "EWJ_Japan_vol_20d", "3M_ret_5d", "AMZN_ret_5d", "Nikkei_Japan_vol_20d", "EOG_EOGResources_vol_20d", "EWQ_France_zscore_60d", "TGT_Target_zscore_60d", "CTAS_Cintas_vol_20d", "SO_SouthernCo_ret_5d", "spx_vol_5d", "XOM_ret_1d", "EOG_EOGResources_ret_5d", "GD_GeneralDynamics_zscore_60d", "LUV_SouthwestAir_ret_5d", "Core_CPI_zscore_60d", "NOC_Northrop_ret_20d", "SCHW_Schwab_ret_5d", "US3Y_Rate_ret_5d", "CLX_Clorox_vol_20d", "WTI_Oil_FRED_zscore_60d", "PAYX_Paychex_zscore_60d"], "is_new": true}, {"model_id": "new_h3_GLOBAL_XGBoost_N25_t2", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 3, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "SBUX_zscore_60d", "XLY_Disc_vol_20d", "CI_Cigna_vol_20d", "CPB_CampbellSoup_ret_20d", "PPL_PPL_ret_1d", "DHR_vol_20d", "XLK_Tech_zscore_60d", "DOW_Price_zscore_60d", "EWQ_France_ret_20d", "EWM_Malaysia_vol_20d", "IWM_SmallCap_vol_20d", "US3M_Rate_vol_20d", "DAX_Germany_vol_20d", "EWY_Korea_ret_20d", "EWH_HongKong_ret_5d", "LOW_Lowes_ret_5d", "EWG_Germany_vol_20d", "spx_vol_5d", "MRK_Merck_zscore_60d", "vix_acceleration_1d", "LMT_LockheedMartin_vol_20d", "Nikkei_Japan_zscore_60d", "ENB_EnbridgeInc_ret_1d"], "is_new": true}, {"model_id": "new_h3_GLOBAL_XGBoost_N25_t3", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 3, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "LMT_LockheedMartin_vol_20d", "heston_ev_h3", "AMT_AmericanTower_ret_1d", "GE_ret_1d", "AORD_AUS_zscore_60d", "US5Y_Rate_ret_5d", "EWY_Korea_zscore_60d", "CPB_CampbellSoup_vol_20d", "GD_GeneralDynamics_zscore_60d", "PAYX_Paychex_vol_20d", "SJM_JM_Smucker_ret_1d", "CMCSA_ret_1d", "Nikkei_Japan_vol_20d", "T_ret_1d", "ENB_EnbridgeInc_ret_1d", "TGT_Target_zscore_60d", "TM_Telephone_ret_1d", "EOG_EOGResources_vol_20d", "Brent_Oil_FRED_ret_5d", "XLF_Fin_vol_20d", "DAX_Germany_zscore_60d", "HD_zscore_60d", "HD_ret_5d"], "is_new": true}, {"model_id": "new_h3_GLOBAL_XGBoost_N25_t4", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 3, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWA_Australia_zscore_60d", "MSTR_Bitcoin3_ret_1d", "DE_Deere_ret_5d", "WTI_Oil_FRED_zscore_60d", "SBUX_vol_20d", "3M_vol_20d", "EWJ_Japan_vol_20d", "T10Y2Y_Spread_ret_5d", "Nikkei_Japan_zscore_60d", "HD_ret_20d", "ORCL_zscore_60d", "EWM_Malaysia_zscore_60d", "DHR_vol_20d", "EMR_Emerson_ret_20d", "EWL_Switzerland_vol_20d", "DHR_ret_1d", "XLK_Tech_zscore_60d", "EWY_Korea_zscore_60d", "EFFR_vol_20d", "heston_var_ev_h3", "EWG_Germany_ret_20d", "PAYX_Paychex_zscore_60d", "US1Y_Rate_ret_5d"], "is_new": true}, {"model_id": "new_h3_GLOBAL_XGBoost_N25_t5", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 3, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EFFR_vol_20d", "JNJ_ret_1d", "EWY_Korea_zscore_60d", "CPB_CampbellSoup_vol_20d", "HUM_Humana_ret_5d", "vix_acceleration_1d", "DE_Deere_vol_20d", "T10Y2Y_Spread_ret_5d", "SBUX_vol_20d", "TM_Telephone_vol_20d", "HD_ret_1d", "Retail_Sales_zscore_60d", "LLY_zscore_60d", "XLK_Tech_zscore_60d", "3M_vol_20d", "gjr_condvar_h1", "EXC_Exelon_zscore_60d", "PLD_Prologis_ret_5d", "HD_ret_20d", "CMCSA_ret_1d", "PAYX_Paychex_ret_20d", "NWL_Newell_ret_20d", "IBEX_Spain_ret_20d"], "is_new": true}, {"model_id": "new_h3_GLOBAL_XGBoost_N25_t6", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 3, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "3M_ret_5d", "LLY_zscore_60d", "PFE_ret_1d", "VOD_Vodafone_zscore_60d", "HangSeng_HK_ret_1d", "ASX_Australia_ret_5d", "MSTR_Bitcoin3_ret_5d", "ITT_ITTInc_ret_5d", "EQR_Equity_ret_1d", "LMT_LockheedMartin_ret_1d", "EWL_Switzerland_vol_20d", "EQIX_Equinix_ret_5d", "US6M_Rate_ret_20d", "EWY_Korea_ret_20d", "DAX_Germany_zscore_60d", "GD_GeneralDynamics_zscore_60d", "spx_abs_ret_max_5d", "XLK_Tech_zscore_60d", "TM_Telephone_ret_1d", "EWM_Malaysia_zscore_60d", "HD_ret_1d", "MRK_Merck_zscore_60d", "Core_PCE_zscore_60d"], "is_new": true}, {"model_id": "new_h3_GLOBAL_XGBoost_N25_t7", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 3, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "T_ret_1d", "hmm_p_stress", "XLB_Materials_zscore_60d", "PAYX_Paychex_zscore_60d", "SBUX_ret_5d", "INTC_ret_5d", "FedFunds_zscore_60d", "SCHW_Schwab_ret_5d", "DE_Deere_vol_20d", "TXN_vol_20d", "Core_PCE_zscore_60d", "ITT_ITTInc_ret_5d", "spx_momentum_3d", "AORD_AUS_zscore_60d", "Nikkei_Japan_vol_20d", "NWL_Newell_ret_20d", "EQIX_Equinix_ret_5d", "LLY_zscore_60d", "3M_ret_5d", "CPB_CampbellSoup_vol_20d", "XLV_Health_zscore_60d", "EOG_EOGResources_vol_20d", "ES_Evergy_ret_1d"], "is_new": true}, {"model_id": "new_h3_GLOBAL_XGBoost_N30_t0", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 3, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "PAYX_Paychex_ret_20d", "DAX_Germany_vol_20d", "EXC_Exelon_zscore_60d", "XLK_Tech_zscore_60d", "HD_ret_5d", "T_ret_1d", "QQQ_vol_20d", "ENB_EnbridgeInc_ret_1d", "HD_ret_20d", "SPY_zscore_60d", "EWY_Korea_zscore_60d", "US7Y_Rate_ret_20d", "DE_Deere_ret_5d", "Nikkei_Japan_vol_20d", "Core_CPI_zscore_60d", "LMT_LockheedMartin_ret_1d", "EOG_EOGResources_vol_20d", "heston_ev_h3", "EWQ_France_ret_20d", "3M_vol_20d", "ITT_ITTInc_ret_5d", "NWL_Newell_ret_20d", "hmm_p_stress", "DE_Deere_vol_20d", "BA_ret_1d", "US30Y_Rate_ret_20d", "AXP_Amex_vol_20d", "SLB_Schlumberger_ret_5d"], "is_new": true}, {"model_id": "new_h3_GLOBAL_XGBoost_N30_t1", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 3, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "VVIX_ret_20d", "SJM_JM_Smucker_ret_5d", "SO_SouthernCo_ret_5d", "EQR_Equity_ret_1d", "EOG_EOGResources_ret_5d", "ASX_Australia_vol_20d", "US6M_Rate_ret_20d", "DE_Deere_ret_5d", "LOW_Lowes_ret_20d", "EQIX_Equinix_ret_5d", "XOM_ret_20d", "Core_CPI_zscore_60d", "3M_vol_20d", "VRP_ma5", "GILD_Gilead_ret_20d", "DHR_vol_20d", "EWC_Canada_zscore_60d", "LLY_zscore_60d", "EXC_Exelon_zscore_60d", "SBUX_zscore_60d", "US3M_Rate_vol_20d", "CCI_CrownCastle_vol_20d", "EWJ_Japan_vol_20d", "AXP_Amex_vol_20d", "EFFR_vol_20d", "vix_mean_abs_ret_5d", "heston_var_ev_h5", "AMD_ret_1d"], "is_new": true}, {"model_id": "new_h3_GLOBAL_XGBoost_N30_t2", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 3, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "AMD_ret_5d", "ENB_EnbridgeInc_ret_1d", "EOG_EOGResources_ret_5d", "Retail_Sales_zscore_60d", "WTI_Oil_FRED_zscore_60d", "US30Y_Rate_ret_20d", "DE_Deere_vol_20d", "LOW_Lowes_ret_5d", "PG_ret_20d", "US3Y_Rate_ret_5d", "heston_ev_h3", "SO_SouthernCo_ret_5d", "TED_Spread_vol_20d", "EWY_Korea_ret_20d", "IYR_US_REIT2_zscore_60d", "XLK_Tech_zscore_60d", "EWY_Korea_zscore_60d", "EWJ_Japan_vol_20d", "spx_vol_5d", "Nikkei_Japan_zscore_60d", "ASX_Australia_ret_5d", "EWM_Malaysia_zscore_60d", "EQIX_Equinix_ret_5d", "spx_abs_ret_max_5d", "HangSeng_HK_ret_1d", "AXP_Amex_vol_20d", "US1Y_Rate_ret_20d", "3M_vol_20d"], "is_new": true}, {"model_id": "new_h3_GLOBAL_XGBoost_N30_t3", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 3, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "AMD_ret_5d", "EWM_Malaysia_zscore_60d", "SBUX_ret_5d", "TGT_Target_zscore_60d", "CPB_CampbellSoup_vol_20d", "SBUX_zscore_60d", "CPB_CampbellSoup_zscore_60d", "LLY_zscore_60d", "AXP_Amex_vol_20d", "MSTR_Bitcoin3_ret_20d", "TM_Telephone_vol_20d", "IWM_SmallCap_vol_20d", "DIS_vol_20d", "HUM_Humana_ret_5d", "heston_ev_h3", "vix_mean_abs_ret_5d", "SLB_Schlumberger_ret_1d", "SLB_Schlumberger_ret_5d", "US30Y_Rate_ret_20d", "US7Y_Rate_ret_20d", "MO_AltriaMG_ret_1d", "US5Y_Rate_ret_5d", "EMR_Emerson_ret_20d", "heston_var_ev_h3", "gjr_condvar_h1", "XOM_ret_20d", "Core_CPI_zscore_60d", "LOW_Lowes_ret_20d"], "is_new": true}, {"model_id": "new_h3_GLOBAL_XGBoost_N30_t4", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 3, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "MSTR_Bitcoin3_ret_5d", "AXP_Amex_ret_20d", "AMT_AmericanTower_ret_1d", "US3Y_Rate_ret_5d", "US7Y_Rate_ret_20d", "XOM_ret_1d", "PCAR_PaccarInc_ret_5d", "HD_zscore_60d", "US5Y_Rate_ret_5d", "HD_ret_1d", "CTAS_Cintas_vol_20d", "CPB_CampbellSoup_vol_20d", "PAYX_Paychex_ret_20d", "gjr_condvar_h1", "MRK_Merck_zscore_60d", "spx_abs_ret_max_5d", "BLK_BlackRock_zscore_60d", "EWQ_France_ret_20d", "Michigan_Sentiment_ret_20d", "HUM_Humana_ret_5d", "SJM_JM_Smucker_ret_5d", "SBUX_zscore_60d", "EWY_Korea_ret_20d", "PPL_PPL_ret_1d", "DAX_Germany_zscore_60d", "LLY_zscore_60d", "spx_vol_5d", "CPB_CampbellSoup_zscore_60d"], "is_new": true}, {"model_id": "new_h3_GLOBAL_XGBoost_N30_t5", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 3, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "T10Y2Y_Spread_ret_5d", "EWG_Germany_vol_20d", "GD_GeneralDynamics_zscore_60d", "EWM_Malaysia_vol_20d", "DE_Deere_vol_20d", "US3M_Rate_vol_20d", "PAYX_Paychex_ret_20d", "TM_Telephone_vol_20d", "Nikkei_Japan_vol_20d", "XOM_ret_20d", "SBUX_zscore_60d", "Brent_Oil_FRED_ret_5d", "IYR_US_REIT2_zscore_60d", "IBEX_Spain_ret_20d", "ES_Evergy_ret_1d", "US3M_Rate_zscore_60d", "Industrial_Production_zscore_60d", "VVIX_ret_20d", "ITT_ITTInc_ret_5d", "SBUX_vol_20d", "CCI_CrownCastle_vol_20d", "US6M_Rate_ret_20d", "HangSeng_HK_ret_5d", "ORCL_zscore_60d", "SJM_JM_Smucker_ret_1d", "BA_ret_1d", "AXP_Amex_ret_20d", "PLD_Prologis_ret_5d"], "is_new": true}, {"model_id": "new_h3_GLOBAL_XGBoost_N30_t6", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 3, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "LUV_SouthwestAir_ret_5d", "EWA_Australia_ret_1d", "EWL_Switzerland_zscore_60d", "M_Macys_vol_20d", "EWM_Malaysia_ret_1d", "LMT_LockheedMartin_ret_1d", "XOM_ret_1d", "DOW_Price_zscore_60d", "Core_CPI_zscore_60d", "XOM_ret_20d", "PPL_PPL_ret_1d", "EQIX_Equinix_ret_5d", "EWG_Germany_ret_20d", "GD_GeneralDynamics_zscore_60d", "AMD_ret_1d", "Industrial_Production_zscore_60d", "CLX_Clorox_vol_20d", "MRK_Merck_zscore_60d", "ITT_ITTInc_ret_5d", "TGT_Target_zscore_60d", "TXN_vol_20d", "ORCL_vol_20d", "NVDA_vol_20d", "EWY_Korea_ret_20d", "CTAS_Cintas_vol_20d", "GE_ret_1d", "spx_vol_5d", "TM_Telephone_vol_20d"], "is_new": true}, {"model_id": "new_h3_GLOBAL_XGBoost_N30_t7", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 3, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "BTI_BritishAmerican_ret_20d", "JNJ_ret_1d", "HangSeng_HK_ret_5d", "DIS_vol_20d", "Core_CPI_zscore_60d", "CTAS_Cintas_vol_20d", "T10Y2Y_Spread_ret_5d", "SJM_JM_Smucker_ret_5d", "XLY_Disc_vol_20d", "Industrial_Production_zscore_60d", "XLK_Tech_zscore_60d", "EXC_Exelon_ret_1d", "VVIX_ret_20d", "spx_abs_ret_max_5d", "spx_momentum_3d", "MSTR_Bitcoin3_ret_5d", "IWM_SmallCap_vol_20d", "US6M_Rate_ret_20d", "IYR_US_REIT2_zscore_60d", "Core_PCE_zscore_60d", "QQQ_vol_20d", "EWG_Germany_vol_20d", "EWM_Malaysia_zscore_60d", "MS_MorganStanley_ret_5d", "MS_MorganStanley_ret_1d", "NWL_Newell_ret_20d", "HangSeng_HK_vol_20d", "T_ret_1d"], "is_new": true}, {"model_id": "new_h3_GLOBAL_LightGBM_N5_t0", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 3, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWL_Switzerland_vol_20d", "PAYX_Paychex_zscore_60d", "FedFunds_zscore_60d"], "is_new": true}, {"model_id": "new_h3_GLOBAL_LightGBM_N5_t1", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 3, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "Core_CPI_zscore_60d", "CPB_CampbellSoup_zscore_60d", "3M_vol_20d"], "is_new": true}, {"model_id": "new_h3_GLOBAL_LightGBM_N5_t2", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 3, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "GILD_Gilead_ret_20d", "GE_ret_1d", "EWQ_France_ret_20d"], "is_new": true}, {"model_id": "new_h3_GLOBAL_LightGBM_N5_t3", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 3, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "US3M_Rate_zscore_60d", "INTC_ret_1d", "CCI_CrownCastle_vol_20d"], "is_new": true}, {"model_id": "new_h3_GLOBAL_LightGBM_N5_t4", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 3, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "IWM_SmallCap_vol_20d", "BTI_BritishAmerican_ret_20d", "PAYX_Paychex_zscore_60d"], "is_new": true}, {"model_id": "new_h3_GLOBAL_LightGBM_N5_t5", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 3, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWQ_France_zscore_60d", "M_Macys_vol_20d", "US6M_Rate_ret_20d"], "is_new": true}, {"model_id": "new_h3_GLOBAL_LightGBM_N5_t6", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 3, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "LUV_SouthwestAir_ret_5d", "XLB_Materials_zscore_60d", "EXC_Exelon_ret_1d"], "is_new": true}, {"model_id": "new_h3_GLOBAL_LightGBM_N5_t7", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 3, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "XLV_Health_zscore_60d", "GE_ret_1d", "AXP_Amex_vol_20d"], "is_new": true}, {"model_id": "new_h3_GLOBAL_LightGBM_N8_t0", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 3, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "3M_vol_20d", "NOC_Northrop_ret_20d", "VOD_Vodafone_zscore_60d", "IWM_SmallCap_vol_20d", "M_Macys_vol_20d", "XLK_Tech_zscore_60d"], "is_new": true}, {"model_id": "new_h3_GLOBAL_LightGBM_N8_t1", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 3, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWA_Australia_ret_1d", "CI_Cigna_vol_20d", "MS_MorganStanley_zscore_60d", "PPL_PPL_ret_1d", "VOD_Vodafone_zscore_60d", "EOG_EOGResources_ret_5d"], "is_new": true}, {"model_id": "new_h3_GLOBAL_LightGBM_N8_t2", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 3, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWQ_France_ret_20d", "AXP_Amex_vol_20d", "PAYX_Paychex_vol_20d", "LLY_zscore_60d", "3M_ret_5d", "Industrial_Production_zscore_60d"], "is_new": true}, {"model_id": "new_h3_GLOBAL_LightGBM_N8_t3", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 3, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "AMT_AmericanTower_ret_1d", "SPY_zscore_60d", "Core_CPI_zscore_60d", "BLK_BlackRock_zscore_60d", "MSTR_Bitcoin3_ret_1d", "PAYX_Paychex_vol_20d"], "is_new": true}, {"model_id": "new_h3_GLOBAL_LightGBM_N8_t4", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 3, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "XOM_ret_1d", "MSTR_Bitcoin3_ret_1d", "MO_AltriaMG_ret_1d", "CPB_CampbellSoup_ret_5d", "AMGN_Amgen_ret_1d", "IWM_SmallCap_vol_20d"], "is_new": true}, {"model_id": "new_h3_GLOBAL_LightGBM_N8_t5", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 3, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "IYM_BasicMaterials_ret_20d", "EWL_Switzerland_vol_20d", "SJM_JM_Smucker_ret_1d", "HD_zscore_60d", "US6M_Rate_ret_20d", "GE_ret_1d"], "is_new": true}, {"model_id": "new_h3_GLOBAL_LightGBM_N8_t6", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 3, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "CI_Cigna_vol_20d", "TED_Spread_vol_20d", "LUV_SouthwestAir_ret_5d", "ORCL_vol_20d", "MSTR_Bitcoin3_ret_5d", "IYR_US_REIT2_zscore_60d"], "is_new": true}, {"model_id": "new_h3_GLOBAL_LightGBM_N8_t7", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 3, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "SLB_Schlumberger_ret_5d", "CPB_CampbellSoup_zscore_60d", "DIS_vol_20d", "NFCI_ret_5d", "SPY_zscore_60d", "T_ret_1d"], "is_new": true}, {"model_id": "new_h3_GLOBAL_LightGBM_N10_t0", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 3, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "XOM_ret_20d", "gjr_condvar_h1", "US1Y_Rate_ret_5d", "MSTR_Bitcoin3_ret_1d", "vix_mean_abs_ret_5d", "EWQ_France_zscore_60d", "EQIX_Equinix_ret_5d", "BLK_BlackRock_zscore_60d"], "is_new": true}, {"model_id": "new_h3_GLOBAL_LightGBM_N10_t1", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 3, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "AMT_AmericanTower_ret_1d", "SBUX_zscore_60d", "XOM_ret_20d", "TED_Spread_vol_20d", "Core_PCE_zscore_60d", "PG_ret_20d", "NWL_Newell_ret_20d", "DAX_Germany_zscore_60d"], "is_new": true}, {"model_id": "new_h3_GLOBAL_LightGBM_N10_t2", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 3, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "TED_Spread_zscore_60d", "TXN_vol_20d", "Core_CPI_zscore_60d", "CI_Cigna_vol_20d", "heston_var_ev_h7", "XOM_ret_1d", "hmm_p_stress", "ENB_EnbridgeInc_ret_1d"], "is_new": true}, {"model_id": "new_h3_GLOBAL_LightGBM_N10_t3", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 3, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "CPB_CampbellSoup_ret_5d", "LOW_Lowes_ret_5d", "CPB_CampbellSoup_ret_20d", "TXN_vol_20d", "AORD_AUS_zscore_60d", "PAYX_Paychex_zscore_60d", "SJM_JM_Smucker_ret_5d", "GD_GeneralDynamics_zscore_60d"], "is_new": true}, {"model_id": "new_h3_GLOBAL_LightGBM_N10_t4", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 3, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "PLD_Prologis_ret_5d", "ORCL_vol_20d", "AMGN_Amgen_ret_1d", "BLK_BlackRock_zscore_60d", "EXC_Exelon_ret_1d", "spx_momentum_3d", "DHR_ret_1d", "LLY_zscore_60d"], "is_new": true}, {"model_id": "new_h3_GLOBAL_LightGBM_N10_t5", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 3, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "SJM_JM_Smucker_ret_1d", "CMCSA_ret_1d", "ES_Evergy_ret_1d", "Industrial_Production_zscore_60d", "EXC_Exelon_ret_1d", "HangSeng_HK_ret_1d", "LMT_LockheedMartin_ret_1d", "TXN_vol_20d"], "is_new": true}, {"model_id": "new_h3_GLOBAL_LightGBM_N10_t6", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 3, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "VVIX_ret_20d", "MS_MorganStanley_ret_5d", "BTI_BritishAmerican_ret_20d", "EXC_Exelon_ret_1d", "US3Y_Rate_ret_5d", "XLV_Health_zscore_60d", "AMGN_Amgen_ret_1d", "BTI_BritishAmerican_ret_5d"], "is_new": true}, {"model_id": "new_h3_GLOBAL_LightGBM_N10_t7", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 3, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "IYR_US_REIT2_zscore_60d", "HangSeng_HK_ret_5d", "heston_var_ev_h7", "HangSeng_HK_ret_1d", "IBEX_Spain_ret_20d", "US3Y_Rate_ret_5d", "NWL_Newell_ret_20d", "EOG_EOGResources_ret_5d"], "is_new": true}, {"model_id": "new_h3_GLOBAL_LightGBM_N12_t0", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 3, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "NVDA_vol_20d", "LMT_LockheedMartin_ret_1d", "EWY_Korea_ret_20d", "US1Y_Rate_ret_20d", "XLV_Health_zscore_60d", "PCAR_PaccarInc_ret_5d", "EWL_Switzerland_vol_20d", "US7Y_Rate_ret_20d", "AXP_Amex_vol_20d", "PFE_ret_1d"], "is_new": true}, {"model_id": "new_h3_GLOBAL_LightGBM_N12_t1", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 3, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "LOW_Lowes_ret_20d", "DOW_Price_zscore_60d", "EXC_Exelon_ret_1d", "heston_var_ev_h5", "HangSeng_HK_ret_1d", "EQR_Equity_ret_1d", "AMD_ret_5d", "EWQ_France_zscore_60d", "CPB_CampbellSoup_vol_20d", "ITT_ITTInc_ret_5d"], "is_new": true}, {"model_id": "new_h3_GLOBAL_LightGBM_N12_t2", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 3, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "LLY_zscore_60d", "US1Y_Rate_ret_5d", "BTI_BritishAmerican_ret_20d", "CPB_CampbellSoup_vol_20d", "DE_Deere_ret_5d", "HD_ret_1d", "MO_AltriaMG_ret_1d", "BDX_Becton_Dickinson_ret_20d", "QQQ_vol_20d", "ORCL_zscore_60d"], "is_new": true}, {"model_id": "new_h3_GLOBAL_LightGBM_N12_t3", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 3, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "IWM_SmallCap_vol_20d", "AVB_AvalonBay_zscore_60d", "XOM_ret_1d", "NVDA_vol_20d", "SLB_Schlumberger_ret_5d", "AXP_Amex_vol_20d", "CPB_CampbellSoup_vol_20d", "US3M_Rate_zscore_60d", "DHR_ret_1d", "DIS_vol_20d"], "is_new": true}, {"model_id": "new_h3_GLOBAL_LightGBM_N12_t4", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 3, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EOG_EOGResources_ret_5d", "EWG_Germany_vol_20d", "XLF_Fin_vol_20d", "heston_var_ev_h7", "EWM_Malaysia_vol_20d", "LOW_Lowes_ret_20d", "EQR_Equity_ret_1d", "NVDA_vol_20d", "3M_ret_5d", "CI_Cigna_vol_20d"], "is_new": true}, {"model_id": "new_h3_GLOBAL_LightGBM_N12_t5", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 3, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "JNJ_ret_1d", "IYR_US_REIT2_zscore_60d", "SLB_Schlumberger_ret_1d", "BTI_BritishAmerican_ret_5d", "EWG_Germany_vol_20d", "DAX_Germany_zscore_60d", "ORCL_vol_20d", "CPB_CampbellSoup_ret_5d", "EWA_Australia_ret_1d", "spx_abs_ret_max_5d"], "is_new": true}, {"model_id": "new_h3_GLOBAL_LightGBM_N12_t6", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 3, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "CI_Cigna_vol_20d", "vix_mean_abs_ret_5d", "US1Y_Rate_ret_20d", "DAX_Germany_zscore_60d", "EWJ_Japan_vol_20d", "EWG_Germany_ret_20d", "heston_var_ev_h7", "DE_Deere_ret_5d", "heston_var_ev_h5", "XOM_ret_20d"], "is_new": true}, {"model_id": "new_h3_GLOBAL_LightGBM_N12_t7", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 3, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWM_Malaysia_ret_1d", "AXP_Amex_ret_20d", "GILD_Gilead_ret_20d", "M_Macys_vol_20d", "PPL_PPL_ret_1d", "IBEX_Spain_ret_20d", "AMGN_Amgen_ret_1d", "PCAR_PaccarInc_ret_5d", "CPB_CampbellSoup_ret_20d", "TXN_vol_20d"], "is_new": true}, {"model_id": "new_h3_GLOBAL_LightGBM_N15_t0", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 3, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "CTAS_Cintas_vol_20d", "Brent_Oil_FRED_ret_5d", "JNJ_ret_1d", "Michigan_Sentiment_ret_20d", "HUM_Humana_ret_5d", "SLB_Schlumberger_ret_1d", "EWL_Switzerland_zscore_60d", "ITT_ITTInc_ret_5d", "NOC_Northrop_ret_20d", "EQIX_Equinix_ret_5d", "US1Y_Rate_ret_5d", "PFE_ret_1d", "Industrial_Production_zscore_60d"], "is_new": true}, {"model_id": "new_h3_GLOBAL_LightGBM_N15_t1", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 3, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWQ_France_ret_20d", "MSTR_Bitcoin3_ret_1d", "HD_ret_5d", "US3M_Rate_vol_20d", "DOW_Price_zscore_60d", "CTAS_Cintas_vol_20d", "LLY_zscore_60d", "VOD_Vodafone_zscore_60d", "NOC_Northrop_ret_20d", "EWY_Korea_zscore_60d", "LUV_SouthwestAir_ret_5d", "T10Y2Y_Spread_ret_5d", "PLD_Prologis_ret_5d"], "is_new": true}, {"model_id": "new_h3_GLOBAL_LightGBM_N15_t2", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 3, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "hmm_p_stress", "XLK_Tech_zscore_60d", "EWH_HongKong_ret_5d", "AXP_Amex_vol_20d", "DE_Deere_ret_5d", "EMR_Emerson_ret_20d", "CPB_CampbellSoup_ret_20d", "ORCL_vol_20d", "US1Y_Rate_ret_20d", "SO_SouthernCo_ret_5d", "AXP_Amex_ret_20d", "PLD_Prologis_ret_5d", "3M_ret_5d"], "is_new": true}, {"model_id": "new_h3_GLOBAL_LightGBM_N15_t3", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 3, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "Industrial_Production_zscore_60d", "SBUX_zscore_60d", "FedFunds_zscore_60d", "CLX_Clorox_vol_20d", "DE_Deere_vol_20d", "MO_AltriaMG_ret_1d", "CPB_CampbellSoup_ret_5d", "EQIX_Equinix_ret_5d", "CPB_CampbellSoup_vol_20d", "TXN_vol_20d", "EWA_Australia_ret_1d", "EWA_Australia_zscore_60d", "HD_zscore_60d"], "is_new": true}, {"model_id": "new_h3_GLOBAL_LightGBM_N15_t4", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 3, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "VRP_ma5", "HangSeng_HK_ret_5d", "GD_GeneralDynamics_zscore_60d", "JNJ_ret_1d", "gjr_condvar_h1", "LUV_SouthwestAir_ret_5d", "QQQ_vol_20d", "ORCL_vol_20d", "MS_MorganStanley_ret_1d", "US6M_Rate_ret_20d", "MSTR_Bitcoin3_ret_1d", "DOW_Price_zscore_60d", "LLY_zscore_60d"], "is_new": true}, {"model_id": "new_h3_GLOBAL_LightGBM_N15_t5", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 3, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "CPB_CampbellSoup_vol_20d", "SBUX_vol_20d", "DAX_Germany_vol_20d", "DE_Deere_vol_20d", "AXP_Amex_ret_20d", "EWM_Malaysia_vol_20d", "HangSeng_HK_vol_20d", "LUV_SouthwestAir_ret_5d", "Retail_Sales_zscore_60d", "GD_GeneralDynamics_zscore_60d", "gjr_condvar_h1", "PAYX_Paychex_ret_20d", "JNJ_ret_1d"], "is_new": true}, {"model_id": "new_h3_GLOBAL_LightGBM_N15_t6", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 3, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "3M_vol_20d", "AXP_Amex_vol_20d", "DE_Deere_ret_5d", "ES_Evergy_ret_1d", "HD_zscore_60d", "EQIX_Equinix_ret_5d", "EWM_Malaysia_zscore_60d", "PAYX_Paychex_zscore_60d", "CMCSA_ret_1d", "Michigan_Sentiment_ret_20d", "MSTR_Bitcoin3_ret_5d", "IBEX_Spain_ret_20d", "EWM_Malaysia_vol_20d"], "is_new": true}, {"model_id": "new_h3_GLOBAL_LightGBM_N15_t7", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 3, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "WTI_Oil_FRED_zscore_60d", "XOM_ret_1d", "XLY_Disc_vol_20d", "PAYX_Paychex_zscore_60d", "XOM_ret_20d", "EFFR_vol_20d", "BTI_BritishAmerican_ret_20d", "BTI_BritishAmerican_ret_5d", "INTC_ret_5d", "DE_Deere_vol_20d", "XLV_Health_zscore_60d", "EMR_Emerson_ret_20d", "PAYX_Paychex_ret_20d"], "is_new": true}, {"model_id": "new_h3_GLOBAL_LightGBM_N20_t0", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 3, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "PPL_PPL_ret_1d", "Nikkei_Japan_vol_20d", "gjr_condvar_h1", "EXC_Exelon_ret_1d", "DAX_Germany_zscore_60d", "AXP_Amex_vol_20d", "heston_ev_h3", "Core_CPI_zscore_60d", "CMCSA_ret_1d", "SJM_JM_Smucker_ret_1d", "EWS_Singapore_ret_5d", "Brent_Oil_FRED_ret_5d", "AVB_AvalonBay_zscore_60d", "XLK_Tech_zscore_60d", "AORD_AUS_zscore_60d", "FedFunds_zscore_60d", "MS_MorganStanley_ret_5d", "ASX_Australia_vol_20d"], "is_new": true}, {"model_id": "new_h3_GLOBAL_LightGBM_N20_t1", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 3, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "IBEX_Spain_ret_20d", "US7Y_Rate_ret_20d", "JNJ_ret_1d", "MS_MorganStanley_ret_1d", "NOC_Northrop_ret_20d", "TGT_Target_zscore_60d", "M_Macys_vol_20d", "LLY_zscore_60d", "CLX_Clorox_vol_20d", "AMT_AmericanTower_ret_1d", "EWG_Germany_ret_20d", "T_ret_1d", "VVIX_ret_20d", "LOW_Lowes_ret_20d", "EWH_HongKong_ret_5d", "PPL_PPL_ret_1d", "heston_ev_h3", "EQR_Equity_ret_1d"], "is_new": true}, {"model_id": "new_h3_GLOBAL_LightGBM_N20_t2", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 3, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "WTI_Oil_FRED_zscore_60d", "AMD_ret_1d", "ENB_EnbridgeInc_ret_1d", "JNJ_ret_1d", "EWJ_Japan_vol_20d", "EQR_Equity_ret_1d", "HangSeng_HK_vol_20d", "GD_GeneralDynamics_zscore_60d", "TED_Spread_zscore_60d", "TXN_vol_20d", "AMT_AmericanTower_ret_1d", "SBUX_zscore_60d", "NEE_NextEra_ret_20d", "ORCL_vol_20d", "LOW_Lowes_ret_20d", "PG_ret_20d", "QQQ_vol_20d", "M_Macys_vol_20d"], "is_new": true}, {"model_id": "new_h3_GLOBAL_LightGBM_N20_t3", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 3, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "VVIX_ret_20d", "ENB_EnbridgeInc_ret_1d", "US1Y_Rate_ret_20d", "US3M_Rate_zscore_60d", "CPB_CampbellSoup_ret_5d", "EXC_Exelon_ret_1d", "NWL_Newell_ret_20d", "EWQ_France_ret_20d", "Brent_Oil_FRED_ret_20d", "spx_abs_ret_max_5d", "HUM_Humana_ret_5d", "JNJ_ret_1d", "EXC_Exelon_zscore_60d", "PCAR_PaccarInc_ret_5d", "US6M_Rate_ret_20d", "BTI_BritishAmerican_ret_20d", "Brent_Oil_FRED_ret_5d", "Michigan_Sentiment_ret_20d"], "is_new": true}, {"model_id": "new_h3_GLOBAL_LightGBM_N20_t4", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 3, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "US1Y_Rate_ret_5d", "EFFR_vol_20d", "DIS_vol_20d", "EMR_Emerson_ret_20d", "EWJ_Japan_vol_20d", "heston_var_ev_h5", "GD_GeneralDynamics_zscore_60d", "GE_ret_1d", "heston_ev_h3", "US3Y_Rate_ret_5d", "HD_zscore_60d", "HD_ret_1d", "PCAR_PaccarInc_ret_5d", "vix_acceleration_1d", "HangSeng_HK_ret_5d", "TM_Telephone_ret_1d", "TED_Spread_vol_20d", "LMT_LockheedMartin_vol_20d"], "is_new": true}, {"model_id": "new_h3_GLOBAL_LightGBM_N20_t5", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 3, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "IYM_BasicMaterials_ret_20d", "CCI_CrownCastle_vol_20d", "FedFunds_zscore_60d", "EQIX_Equinix_ret_5d", "Core_PCE_zscore_60d", "gjr_condvar_h1", "MO_AltriaMG_ret_1d", "PG_ret_20d", "DHR_ret_1d", "XLY_Disc_vol_20d", "VOD_Vodafone_zscore_60d", "CMCSA_ret_1d", "EXC_Exelon_ret_1d", "DE_Deere_ret_5d", "MSTR_Bitcoin3_ret_20d", "BDX_Becton_Dickinson_ret_20d", "Retail_Sales_zscore_60d", "BTI_BritishAmerican_ret_20d"], "is_new": true}, {"model_id": "new_h3_GLOBAL_LightGBM_N20_t6", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 3, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWM_Malaysia_zscore_60d", "TGT_Target_zscore_60d", "DE_Deere_ret_5d", "ORCL_zscore_60d", "Michigan_Sentiment_ret_20d", "HD_ret_20d", "spx_vol_5d", "MS_MorganStanley_zscore_60d", "Core_CPI_zscore_60d", "EWJ_Japan_vol_20d", "WTI_Oil_FRED_zscore_60d", "LMT_LockheedMartin_vol_20d", "ITT_ITTInc_ret_5d", "FedFunds_zscore_60d", "LUV_SouthwestAir_ret_5d", "heston_var_ev_h3", "DIS_vol_20d", "IYR_US_REIT2_zscore_60d"], "is_new": true}, {"model_id": "new_h3_GLOBAL_LightGBM_N20_t7", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 3, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EOG_EOGResources_vol_20d", "AMD_ret_1d", "NFCI_ret_5d", "PLD_Prologis_ret_5d", "ORCL_zscore_60d", "heston_var_ev_h3", "vix_acceleration_1d", "EWL_Switzerland_vol_20d", "BLK_BlackRock_zscore_60d", "XLY_Disc_vol_20d", "PFE_ret_1d", "US30Y_Rate_ret_20d", "AMZN_ret_5d", "EWQ_France_zscore_60d", "XLK_Tech_zscore_60d", "EWJ_Japan_vol_20d", "EXC_Exelon_ret_1d", "M_Macys_vol_20d"], "is_new": true}, {"model_id": "new_h3_GLOBAL_LightGBM_N25_t0", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 3, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "Retail_Sales_zscore_60d", "JNJ_ret_1d", "EWJ_Japan_vol_20d", "US3M_Rate_vol_20d", "Core_CPI_zscore_60d", "WTI_Oil_FRED_zscore_60d", "CLX_Clorox_vol_20d", "heston_var_ev_h3", "US7Y_Rate_ret_20d", "EWA_Australia_ret_1d", "T_ret_1d", "EFFR_vol_20d", "SPY_zscore_60d", "spx_momentum_3d", "CI_Cigna_vol_20d", "SLB_Schlumberger_ret_5d", "EWH_HongKong_ret_5d", "ITT_ITTInc_ret_5d", "AMD_ret_5d", "PAYX_Paychex_ret_20d", "EWQ_France_ret_20d", "SJM_JM_Smucker_ret_5d", "HangSeng_HK_ret_5d"], "is_new": true}, {"model_id": "new_h3_GLOBAL_LightGBM_N25_t1", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 3, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "LOW_Lowes_ret_5d", "ASX_Australia_vol_20d", "MS_MorganStanley_ret_5d", "SLB_Schlumberger_ret_5d", "Core_PCE_zscore_60d", "LLY_zscore_60d", "CPB_CampbellSoup_ret_20d", "CLX_Clorox_vol_20d", "FedFunds_zscore_60d", "NOC_Northrop_ret_20d", "DE_Deere_vol_20d", "PFE_ret_1d", "SJM_JM_Smucker_ret_5d", "IYR_US_REIT2_zscore_60d", "MS_MorganStanley_zscore_60d", "NEE_NextEra_ret_20d", "EMR_Emerson_ret_20d", "Brent_Oil_FRED_ret_20d", "PPL_PPL_ret_1d", "spx_vol_5d", "XLF_Fin_vol_20d", "PG_ret_20d", "EQIX_Equinix_ret_5d"], "is_new": true}, {"model_id": "new_h3_GLOBAL_LightGBM_N25_t2", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 3, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "GD_GeneralDynamics_zscore_60d", "LOW_Lowes_ret_5d", "NEE_NextEra_ret_20d", "LMT_LockheedMartin_ret_1d", "CI_Cigna_vol_20d", "vix_mean_abs_ret_5d", "VRP_ma5", "XLB_Materials_zscore_60d", "PAYX_Paychex_zscore_60d", "JNJ_ret_1d", "EWG_Germany_vol_20d", "ASX_Australia_ret_5d", "BTI_BritishAmerican_ret_5d", "HUM_Humana_ret_5d", "EXC_Exelon_zscore_60d", "MSTR_Bitcoin3_ret_5d", "EWJ_Japan_vol_20d", "EWS_Singapore_ret_5d", "TXN_vol_20d", "XLV_Health_zscore_60d", "ITT_ITTInc_ret_5d", "IWM_SmallCap_vol_20d", "HD_ret_20d"], "is_new": true}, {"model_id": "new_h3_GLOBAL_LightGBM_N25_t3", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 3, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "spx_abs_ret_max_5d", "DHR_vol_20d", "DHR_ret_1d", "Brent_Oil_FRED_ret_20d", "XLV_Health_zscore_60d", "T10Y2Y_Spread_ret_5d", "heston_var_ev_h7", "SJM_JM_Smucker_ret_5d", "HangSeng_HK_ret_1d", "US30Y_Rate_ret_20d", "CPB_CampbellSoup_ret_5d", "EWG_Germany_vol_20d", "CCI_CrownCastle_vol_20d", "DAX_Germany_zscore_60d", "BA_ret_1d", "LOW_Lowes_ret_20d", "XLF_Fin_vol_20d", "EWS_Singapore_ret_5d", "BDX_Becton_Dickinson_ret_20d", "US1Y_Rate_ret_20d", "US3Y_Rate_ret_5d", "hmm_p_stress", "EWM_Malaysia_zscore_60d"], "is_new": true}, {"model_id": "new_h3_GLOBAL_LightGBM_N25_t4", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 3, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "AVB_AvalonBay_zscore_60d", "ENB_EnbridgeInc_ret_1d", "IBEX_Spain_ret_20d", "BTI_BritishAmerican_ret_5d", "SJM_JM_Smucker_ret_1d", "Nikkei_Japan_vol_20d", "NEE_NextEra_ret_20d", "VRP_ma5", "ORCL_zscore_60d", "AXP_Amex_ret_20d", "Michigan_Sentiment_ret_20d", "TED_Spread_zscore_60d", "EWG_Germany_ret_20d", "T_ret_1d", "EFFR_vol_20d", "US1Y_Rate_ret_20d", "vix_acceleration_1d", "MSTR_Bitcoin3_ret_1d", "heston_var_ev_h7", "PAYX_Paychex_vol_20d", "HD_zscore_60d", "BLK_BlackRock_zscore_60d", "AMGN_Amgen_ret_1d"], "is_new": true}, {"model_id": "new_h3_GLOBAL_LightGBM_N25_t5", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 3, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWL_Switzerland_zscore_60d", "IWM_SmallCap_vol_20d", "HUM_Humana_ret_5d", "US6M_Rate_ret_20d", "NEE_NextEra_ret_20d", "ES_Evergy_ret_1d", "LUV_SouthwestAir_ret_5d", "WTI_Oil_FRED_zscore_60d", "LMT_LockheedMartin_ret_1d", "VRP_ma5", "AMGN_Amgen_ret_1d", "Core_CPI_zscore_60d", "DHR_vol_20d", "CMCSA_ret_1d", "EWL_Switzerland_vol_20d", "heston_var_ev_h7", "IYR_US_REIT2_zscore_60d", "EWA_Australia_ret_1d", "BA_ret_1d", "Core_PCE_zscore_60d", "EMR_Emerson_ret_20d", "Industrial_Production_zscore_60d", "vix_mean_abs_ret_5d"], "is_new": true}, {"model_id": "new_h3_GLOBAL_LightGBM_N25_t6", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 3, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "MS_MorganStanley_ret_1d", "Nikkei_Japan_vol_20d", "US6M_Rate_ret_20d", "EWL_Switzerland_zscore_60d", "PLD_Prologis_ret_5d", "BLK_BlackRock_zscore_60d", "vix_acceleration_1d", "QQQ_vol_20d", "heston_ev_h3", "IYM_BasicMaterials_ret_20d", "ORCL_vol_20d", "HangSeng_HK_ret_1d", "HD_zscore_60d", "AMT_AmericanTower_ret_1d", "EWH_HongKong_ret_5d", "CLX_Clorox_vol_20d", "GD_GeneralDynamics_zscore_60d", "SO_SouthernCo_ret_5d", "EOG_EOGResources_vol_20d", "EQR_Equity_ret_1d", "Core_PCE_zscore_60d", "Brent_Oil_FRED_ret_20d", "EXC_Exelon_zscore_60d"], "is_new": true}, {"model_id": "new_h3_GLOBAL_LightGBM_N25_t7", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 3, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "LLY_zscore_60d", "MRK_Merck_zscore_60d", "INTC_ret_5d", "EWM_Malaysia_vol_20d", "EMR_Emerson_ret_20d", "CPB_CampbellSoup_ret_5d", "MS_MorganStanley_zscore_60d", "US1Y_Rate_ret_20d", "IWM_SmallCap_vol_20d", "LMT_LockheedMartin_vol_20d", "spx_vol_5d", "3M_ret_5d", "EFFR_ret_1d", "MS_MorganStanley_ret_1d", "VVIX_ret_20d", "HD_ret_20d", "MO_AltriaMG_ret_1d", "AMD_ret_5d", "heston_ev_h3", "T10Y2Y_Spread_ret_5d", "EWH_HongKong_ret_5d", "MS_MorganStanley_ret_5d", "BA_ret_1d"], "is_new": true}, {"model_id": "new_h3_GLOBAL_LightGBM_N30_t0", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 3, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "US6M_Rate_ret_20d", "US5Y_Rate_ret_5d", "EWG_Germany_ret_20d", "vix_mean_abs_ret_5d", "DOW_Price_zscore_60d", "TGT_Target_zscore_60d", "heston_ev_h3", "US30Y_Rate_ret_20d", "NEE_NextEra_ret_20d", "MS_MorganStanley_ret_1d", "SLB_Schlumberger_ret_1d", "spx_momentum_3d", "TED_Spread_zscore_60d", "AMD_ret_5d", "CPB_CampbellSoup_ret_5d", "AORD_AUS_zscore_60d", "M_Macys_vol_20d", "JNJ_ret_1d", "EQR_Equity_ret_1d", "XOM_ret_1d", "LLY_zscore_60d", "HD_ret_20d", "IBEX_Spain_ret_20d", "EWC_Canada_zscore_60d", "ASX_Australia_ret_5d", "INTC_ret_1d", "EOG_EOGResources_vol_20d", "EWJ_Japan_vol_20d"], "is_new": true}, {"model_id": "new_h3_GLOBAL_LightGBM_N30_t1", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 3, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "CPB_CampbellSoup_ret_20d", "Nikkei_Japan_zscore_60d", "EWM_Malaysia_zscore_60d", "spx_abs_ret_max_5d", "PAYX_Paychex_ret_20d", "AMGN_Amgen_ret_1d", "PFE_ret_1d", "heston_var_ev_h3", "EMR_Emerson_ret_20d", "SJM_JM_Smucker_ret_1d", "EXC_Exelon_zscore_60d", "DHR_ret_1d", "AORD_AUS_zscore_60d", "AXP_Amex_ret_20d", "LMT_LockheedMartin_ret_1d", "BA_ret_1d", "EQR_Equity_ret_1d", "HUM_Humana_ret_5d", "3M_vol_20d", "spx_momentum_3d", "US3M_Rate_zscore_60d", "EWM_Malaysia_vol_20d", "VVIX_ret_20d", "QQQ_vol_20d", "US30Y_Rate_ret_20d", "SO_SouthernCo_ret_5d", "CI_Cigna_vol_20d", "AXP_Amex_vol_20d"], "is_new": true}, {"model_id": "new_h3_GLOBAL_LightGBM_N30_t2", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 3, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "VVIX_ret_20d", "AMZN_ret_5d", "LOW_Lowes_ret_20d", "FedFunds_zscore_60d", "EOG_EOGResources_ret_5d", "HUM_Humana_ret_5d", "US1Y_Rate_ret_5d", "PAYX_Paychex_vol_20d", "T10Y2Y_Spread_ret_5d", "HD_ret_20d", "AXP_Amex_ret_20d", "HangSeng_HK_vol_20d", "CPB_CampbellSoup_vol_20d", "XOM_ret_20d", "SJM_JM_Smucker_ret_1d", "XOM_ret_1d", "EXC_Exelon_zscore_60d", "3M_vol_20d", "SO_SouthernCo_ret_5d", "SPY_zscore_60d", "PG_ret_20d", "SBUX_vol_20d", "US5Y_Rate_ret_5d", "LMT_LockheedMartin_ret_1d", "Brent_Oil_FRED_ret_5d", "ITT_ITTInc_ret_5d", "M_Macys_vol_20d", "EMR_Emerson_ret_20d"], "is_new": true}, {"model_id": "new_h3_GLOBAL_LightGBM_N30_t3", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 3, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "gjr_condvar_h1", "US7Y_Rate_ret_20d", "ASX_Australia_ret_5d", "XOM_ret_20d", "BTI_BritishAmerican_ret_20d", "MS_MorganStanley_zscore_60d", "US1Y_Rate_ret_20d", "TED_Spread_vol_20d", "HD_ret_20d", "AMD_ret_5d", "HangSeng_HK_vol_20d", "Retail_Sales_zscore_60d", "Michigan_Sentiment_ret_20d", "EWQ_France_ret_20d", "EWM_Malaysia_vol_20d", "AMD_ret_1d", "US6M_Rate_ret_20d", "IBEX_Spain_ret_20d", "PPL_PPL_ret_1d", "VVIX_ret_20d", "CPB_CampbellSoup_ret_5d", "vix_mean_abs_ret_5d", "CPB_CampbellSoup_vol_20d", "INTC_ret_1d", "US5Y_Rate_ret_5d", "ES_Evergy_ret_1d", "SJM_JM_Smucker_ret_5d", "HangSeng_HK_ret_5d"], "is_new": true}, {"model_id": "new_h3_GLOBAL_LightGBM_N30_t4", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 3, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "PCAR_PaccarInc_ret_5d", "HD_ret_1d", "CPB_CampbellSoup_vol_20d", "Nikkei_Japan_zscore_60d", "Industrial_Production_zscore_60d", "EWQ_France_zscore_60d", "PAYX_Paychex_ret_20d", "vix_acceleration_1d", "NEE_NextEra_ret_20d", "MSTR_Bitcoin3_ret_1d", "ORCL_zscore_60d", "US1Y_Rate_ret_5d", "JNJ_ret_1d", "NVDA_vol_20d", "EWG_Germany_vol_20d", "heston_var_ev_h7", "CPB_CampbellSoup_ret_5d", "heston_ev_h3", "HUM_Humana_ret_5d", "EWQ_France_ret_20d", "MS_MorganStanley_zscore_60d", "BTI_BritishAmerican_ret_5d", "Core_CPI_zscore_60d", "CPB_CampbellSoup_ret_20d", "Michigan_Sentiment_ret_20d", "SPY_zscore_60d", "MO_AltriaMG_ret_1d", "MRK_Merck_zscore_60d"], "is_new": true}, {"model_id": "new_h3_GLOBAL_LightGBM_N30_t5", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 3, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "Core_PCE_zscore_60d", "Nikkei_Japan_zscore_60d", "T10Y2Y_Spread_ret_5d", "SCHW_Schwab_ret_5d", "HangSeng_HK_ret_5d", "SBUX_zscore_60d", "T_ret_1d", "EMR_Emerson_ret_20d", "INTC_ret_1d", "NEE_NextEra_ret_20d", "EWY_Korea_ret_20d", "GE_ret_1d", "EWG_Germany_vol_20d", "EOG_EOGResources_vol_20d", "IWM_SmallCap_vol_20d", "CLX_Clorox_vol_20d", "AMD_ret_1d", "DAX_Germany_vol_20d", "CCI_CrownCastle_vol_20d", "AMZN_ret_5d", "ENB_EnbridgeInc_ret_1d", "US1Y_Rate_ret_5d", "spx_vol_5d", "EWG_Germany_ret_20d", "vix_mean_abs_ret_5d", "GD_GeneralDynamics_zscore_60d", "BTI_BritishAmerican_ret_5d", "XOM_ret_20d"], "is_new": true}, {"model_id": "new_h3_GLOBAL_LightGBM_N30_t6", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 3, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "DE_Deere_ret_5d", "AXP_Amex_ret_20d", "CCI_CrownCastle_vol_20d", "heston_var_ev_h7", "PCAR_PaccarInc_ret_5d", "HangSeng_HK_vol_20d", "PLD_Prologis_ret_5d", "EXC_Exelon_zscore_60d", "US30Y_Rate_ret_20d", "CMCSA_ret_1d", "INTC_ret_1d", "QQQ_vol_20d", "US3M_Rate_zscore_60d", "ORCL_vol_20d", "CPB_CampbellSoup_ret_20d", "VOD_Vodafone_zscore_60d", "GILD_Gilead_ret_20d", "vix_mean_abs_ret_5d", "EWA_Australia_ret_1d", "SJM_JM_Smucker_ret_5d", "BDX_Becton_Dickinson_ret_20d", "ES_Evergy_ret_1d", "BTI_BritishAmerican_ret_5d", "EOG_EOGResources_vol_20d", "EXC_Exelon_ret_1d", "ITT_ITTInc_ret_5d", "TM_Telephone_ret_1d", "EFFR_ret_1d"], "is_new": true}, {"model_id": "new_h3_GLOBAL_LightGBM_N30_t7", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 3, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "SJM_JM_Smucker_ret_1d", "AXP_Amex_ret_20d", "US7Y_Rate_ret_20d", "EQR_Equity_ret_1d", "PG_ret_20d", "SPY_zscore_60d", "heston_var_ev_h5", "LOW_Lowes_ret_5d", "PAYX_Paychex_vol_20d", "GD_GeneralDynamics_zscore_60d", "EWA_Australia_zscore_60d", "NEE_NextEra_ret_20d", "HD_ret_5d", "EMR_Emerson_ret_20d", "IYM_BasicMaterials_ret_20d", "GILD_Gilead_ret_20d", "EQIX_Equinix_ret_5d", "IBEX_Spain_ret_20d", "spx_abs_ret_max_5d", "BLK_BlackRock_zscore_60d", "EOG_EOGResources_vol_20d", "MSTR_Bitcoin3_ret_5d", "CPB_CampbellSoup_zscore_60d", "INTC_ret_5d", "HD_zscore_60d", "M_Macys_vol_20d", "Michigan_Sentiment_ret_20d", "ORCL_vol_20d"], "is_new": true}, {"model_id": "new_h3_GLOBAL_GradientBoosting_N5_t0", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 3, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EOG_EOGResources_vol_20d", "US3Y_Rate_ret_5d", "PFE_ret_1d"], "is_new": true}, {"model_id": "new_h3_GLOBAL_GradientBoosting_N5_t1", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 3, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "T10Y2Y_Spread_ret_5d", "heston_var_ev_h7", "ORCL_zscore_60d"], "is_new": true}, {"model_id": "new_h3_GLOBAL_GradientBoosting_N5_t2", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 3, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "HangSeng_HK_ret_5d", "GILD_Gilead_ret_20d", "LMT_LockheedMartin_ret_1d"], "is_new": true}, {"model_id": "new_h3_GLOBAL_GradientBoosting_N5_t3", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 3, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "T_ret_1d", "EOG_EOGResources_vol_20d", "US3M_Rate_zscore_60d"], "is_new": true}, {"model_id": "new_h3_GLOBAL_GradientBoosting_N5_t4", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 3, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "MS_MorganStanley_ret_1d", "VVIX_ret_20d", "SJM_JM_Smucker_ret_5d"], "is_new": true}, {"model_id": "new_h3_GLOBAL_GradientBoosting_N5_t5", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 3, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "3M_ret_5d", "EFFR_ret_1d", "PAYX_Paychex_zscore_60d"], "is_new": true}, {"model_id": "new_h3_GLOBAL_GradientBoosting_N5_t6", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 3, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "spx_momentum_3d", "MSTR_Bitcoin3_ret_1d", "US1Y_Rate_ret_20d"], "is_new": true}, {"model_id": "new_h3_GLOBAL_GradientBoosting_N5_t7", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 3, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "HUM_Humana_ret_5d", "PFE_ret_1d", "WTI_Oil_FRED_zscore_60d"], "is_new": true}, {"model_id": "new_h3_GLOBAL_GradientBoosting_N8_t0", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 3, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "IWM_SmallCap_vol_20d", "CCI_CrownCastle_vol_20d", "LLY_zscore_60d", "EWQ_France_ret_20d", "SBUX_zscore_60d", "heston_ev_h3"], "is_new": true}, {"model_id": "new_h3_GLOBAL_GradientBoosting_N8_t1", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 3, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "Nikkei_Japan_zscore_60d", "AMT_AmericanTower_ret_1d", "TM_Telephone_vol_20d", "DE_Deere_ret_5d", "EWY_Korea_ret_20d", "SPY_zscore_60d"], "is_new": true}, {"model_id": "new_h3_GLOBAL_GradientBoosting_N8_t2", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 3, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "heston_ev_h3", "EOG_EOGResources_vol_20d", "VVIX_ret_20d", "PPL_PPL_ret_1d", "TM_Telephone_ret_1d", "AMD_ret_1d"], "is_new": true}, {"model_id": "new_h3_GLOBAL_GradientBoosting_N8_t3", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 3, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "NWL_Newell_ret_20d", "INTC_ret_1d", "gjr_condvar_h1", "EWG_Germany_ret_20d", "3M_vol_20d", "XLK_Tech_zscore_60d"], "is_new": true}, {"model_id": "new_h3_GLOBAL_GradientBoosting_N8_t4", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 3, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "Core_PCE_zscore_60d", "DAX_Germany_zscore_60d", "SO_SouthernCo_ret_5d", "T_ret_1d", "CPB_CampbellSoup_vol_20d", "M_Macys_vol_20d"], "is_new": true}, {"model_id": "new_h3_GLOBAL_GradientBoosting_N8_t5", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 3, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "TM_Telephone_ret_1d", "EWL_Switzerland_vol_20d", "EOG_EOGResources_vol_20d", "DOW_Price_zscore_60d", "EWM_Malaysia_vol_20d", "ENB_EnbridgeInc_ret_1d"], "is_new": true}, {"model_id": "new_h3_GLOBAL_GradientBoosting_N8_t6", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 3, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EXC_Exelon_zscore_60d", "DHR_ret_1d", "GD_GeneralDynamics_zscore_60d", "Brent_Oil_FRED_ret_20d", "MS_MorganStanley_ret_5d", "BDX_Becton_Dickinson_ret_20d"], "is_new": true}, {"model_id": "new_h3_GLOBAL_GradientBoosting_N8_t7", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 3, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "TXN_vol_20d", "US7Y_Rate_ret_20d", "heston_var_ev_h7", "VVIX_ret_20d", "GE_ret_1d", "TM_Telephone_vol_20d"], "is_new": true}, {"model_id": "new_h3_GLOBAL_GradientBoosting_N10_t0", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 3, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWA_Australia_zscore_60d", "DAX_Germany_zscore_60d", "EQIX_Equinix_ret_5d", "SJM_JM_Smucker_ret_5d", "HangSeng_HK_ret_5d", "NWL_Newell_ret_20d", "MS_MorganStanley_zscore_60d", "Core_PCE_zscore_60d"], "is_new": true}, {"model_id": "new_h3_GLOBAL_GradientBoosting_N10_t1", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 3, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "SLB_Schlumberger_ret_5d", "CPB_CampbellSoup_ret_20d", "M_Macys_vol_20d", "EWC_Canada_zscore_60d", "DIS_vol_20d", "CTAS_Cintas_vol_20d", "EWQ_France_zscore_60d", "heston_var_ev_h5"], "is_new": true}, {"model_id": "new_h3_GLOBAL_GradientBoosting_N10_t2", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 3, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "CCI_CrownCastle_vol_20d", "EXC_Exelon_zscore_60d", "LLY_zscore_60d", "spx_vol_5d", "LOW_Lowes_ret_20d", "HangSeng_HK_ret_5d", "EXC_Exelon_ret_1d", "PAYX_Paychex_vol_20d"], "is_new": true}, {"model_id": "new_h3_GLOBAL_GradientBoosting_N10_t3", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 3, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "BA_ret_1d", "FedFunds_zscore_60d", "SPY_zscore_60d", "XLK_Tech_zscore_60d", "heston_var_ev_h3", "XLY_Disc_vol_20d", "SCHW_Schwab_ret_5d", "EXC_Exelon_ret_1d"], "is_new": true}, {"model_id": "new_h3_GLOBAL_GradientBoosting_N10_t4", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 3, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWG_Germany_ret_20d", "ASX_Australia_ret_5d", "EWL_Switzerland_vol_20d", "SBUX_zscore_60d", "EWY_Korea_zscore_60d", "SO_SouthernCo_ret_5d", "LUV_SouthwestAir_ret_5d", "heston_var_ev_h7"], "is_new": true}, {"model_id": "new_h3_GLOBAL_GradientBoosting_N10_t5", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 3, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWL_Switzerland_zscore_60d", "DIS_vol_20d", "AXP_Amex_vol_20d", "BA_ret_1d", "M_Macys_vol_20d", "spx_momentum_3d", "3M_vol_20d", "heston_var_ev_h7"], "is_new": true}, {"model_id": "new_h3_GLOBAL_GradientBoosting_N10_t6", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 3, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "SBUX_vol_20d", "EWS_Singapore_ret_5d", "ORCL_vol_20d", "vix_acceleration_1d", "MSTR_Bitcoin3_ret_20d", "SBUX_ret_5d", "LUV_SouthwestAir_ret_5d", "SLB_Schlumberger_ret_1d"], "is_new": true}, {"model_id": "new_h3_GLOBAL_GradientBoosting_N10_t7", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 3, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "INTC_ret_5d", "PLD_Prologis_ret_5d", "EWM_Malaysia_zscore_60d", "EWM_Malaysia_ret_1d", "IYR_US_REIT2_zscore_60d", "EWL_Switzerland_zscore_60d", "SJM_JM_Smucker_ret_5d", "EWJ_Japan_vol_20d"], "is_new": true}, {"model_id": "new_h3_GLOBAL_GradientBoosting_N12_t0", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 3, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "TGT_Target_zscore_60d", "VRP_ma5", "HD_zscore_60d", "BDX_Becton_Dickinson_ret_20d", "IWM_SmallCap_vol_20d", "ORCL_vol_20d", "VVIX_ret_20d", "PFE_ret_1d", "MSTR_Bitcoin3_ret_20d", "CPB_CampbellSoup_vol_20d"], "is_new": true}, {"model_id": "new_h3_GLOBAL_GradientBoosting_N12_t1", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 3, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EFFR_ret_1d", "AXP_Amex_ret_20d", "PG_ret_20d", "XLF_Fin_vol_20d", "gjr_condvar_h1", "M_Macys_vol_20d", "HD_ret_1d", "TM_Telephone_vol_20d", "ASX_Australia_vol_20d", "US5Y_Rate_ret_5d"], "is_new": true}, {"model_id": "new_h3_GLOBAL_GradientBoosting_N12_t2", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 3, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "FedFunds_zscore_60d", "spx_momentum_3d", "DE_Deere_vol_20d", "PCAR_PaccarInc_ret_5d", "IWM_SmallCap_vol_20d", "Nikkei_Japan_vol_20d", "Core_CPI_zscore_60d", "PAYX_Paychex_zscore_60d", "EWS_Singapore_ret_5d", "PAYX_Paychex_vol_20d"], "is_new": true}, {"model_id": "new_h3_GLOBAL_GradientBoosting_N12_t3", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 3, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "AMT_AmericanTower_ret_1d", "BTI_BritishAmerican_ret_20d", "CTAS_Cintas_vol_20d", "PAYX_Paychex_vol_20d", "ITT_ITTInc_ret_5d", "heston_var_ev_h3", "IYM_BasicMaterials_ret_20d", "IYR_US_REIT2_zscore_60d", "MS_MorganStanley_ret_5d", "PCAR_PaccarInc_ret_5d"], "is_new": true}, {"model_id": "new_h3_GLOBAL_GradientBoosting_N12_t4", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 3, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "MO_AltriaMG_ret_1d", "CTAS_Cintas_vol_20d", "AXP_Amex_ret_20d", "NVDA_vol_20d", "PCAR_PaccarInc_ret_5d", "HangSeng_HK_ret_5d", "HangSeng_HK_ret_1d", "EXC_Exelon_zscore_60d", "XLB_Materials_zscore_60d", "XLF_Fin_vol_20d"], "is_new": true}, {"model_id": "new_h3_GLOBAL_GradientBoosting_N12_t5", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 3, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "M_Macys_vol_20d", "SBUX_vol_20d", "XOM_ret_20d", "SBUX_zscore_60d", "SPY_zscore_60d", "SO_SouthernCo_ret_5d", "BA_ret_1d", "EWQ_France_ret_20d", "AMGN_Amgen_ret_1d", "ENB_EnbridgeInc_ret_1d"], "is_new": true}, {"model_id": "new_h3_GLOBAL_GradientBoosting_N12_t6", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 3, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "AMD_ret_1d", "US30Y_Rate_ret_20d", "EWJ_Japan_vol_20d", "EWM_Malaysia_ret_1d", "HangSeng_HK_ret_5d", "Core_PCE_zscore_60d", "TGT_Target_zscore_60d", "Retail_Sales_zscore_60d", "INTC_ret_5d", "INTC_ret_1d"], "is_new": true}, {"model_id": "new_h3_GLOBAL_GradientBoosting_N12_t7", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 3, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "BDX_Becton_Dickinson_ret_20d", "EWM_Malaysia_vol_20d", "SLB_Schlumberger_ret_5d", "AMD_ret_1d", "EWA_Australia_ret_1d", "MO_AltriaMG_ret_1d", "GD_GeneralDynamics_zscore_60d", "EWG_Germany_vol_20d", "CPB_CampbellSoup_vol_20d", "EWY_Korea_ret_20d"], "is_new": true}, {"model_id": "new_h3_GLOBAL_GradientBoosting_N15_t0", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 3, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "US3Y_Rate_ret_5d", "MO_AltriaMG_ret_1d", "AORD_AUS_zscore_60d", "GE_ret_1d", "US1Y_Rate_ret_20d", "SO_SouthernCo_ret_5d", "XOM_ret_20d", "TXN_vol_20d", "CPB_CampbellSoup_zscore_60d", "XLV_Health_zscore_60d", "TM_Telephone_vol_20d", "SJM_JM_Smucker_ret_1d", "LUV_SouthwestAir_ret_5d"], "is_new": true}, {"model_id": "new_h3_GLOBAL_GradientBoosting_N15_t1", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 3, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EQIX_Equinix_ret_5d", "heston_var_ev_h5", "US7Y_Rate_ret_20d", "DE_Deere_vol_20d", "SBUX_ret_5d", "TED_Spread_vol_20d", "ASX_Australia_vol_20d", "XOM_ret_1d", "VRP_ma5", "DHR_vol_20d", "T10Y2Y_Spread_ret_5d", "EWQ_France_ret_20d", "spx_vol_5d"], "is_new": true}, {"model_id": "new_h3_GLOBAL_GradientBoosting_N15_t2", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 3, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "gjr_condvar_h1", "ASX_Australia_vol_20d", "NVDA_vol_20d", "EQIX_Equinix_ret_5d", "AORD_AUS_zscore_60d", "GILD_Gilead_ret_20d", "Nikkei_Japan_vol_20d", "PFE_ret_1d", "US30Y_Rate_ret_20d", "NOC_Northrop_ret_20d", "EWC_Canada_zscore_60d", "AVB_AvalonBay_zscore_60d", "EWL_Switzerland_zscore_60d"], "is_new": true}, {"model_id": "new_h3_GLOBAL_GradientBoosting_N15_t3", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 3, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "US3Y_Rate_ret_5d", "EXC_Exelon_ret_1d", "TXN_vol_20d", "MS_MorganStanley_ret_5d", "Retail_Sales_zscore_60d", "AVB_AvalonBay_zscore_60d", "PLD_Prologis_ret_5d", "hmm_p_stress", "MSTR_Bitcoin3_ret_20d", "XLK_Tech_zscore_60d", "TM_Telephone_ret_1d", "NEE_NextEra_ret_20d", "GE_ret_1d"], "is_new": true}, {"model_id": "new_h3_GLOBAL_GradientBoosting_N15_t4", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 3, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "CCI_CrownCastle_vol_20d", "AVB_AvalonBay_zscore_60d", "CPB_CampbellSoup_zscore_60d", "JNJ_ret_1d", "FedFunds_zscore_60d", "vix_mean_abs_ret_5d", "ENB_EnbridgeInc_ret_1d", "ORCL_zscore_60d", "EOG_EOGResources_vol_20d", "ITT_ITTInc_ret_5d", "IYR_US_REIT2_zscore_60d", "PFE_ret_1d", "EWM_Malaysia_vol_20d"], "is_new": true}, {"model_id": "new_h3_GLOBAL_GradientBoosting_N15_t5", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 3, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "DE_Deere_vol_20d", "Industrial_Production_zscore_60d", "SJM_JM_Smucker_ret_5d", "HangSeng_HK_ret_5d", "EOG_EOGResources_vol_20d", "EWM_Malaysia_ret_1d", "AMGN_Amgen_ret_1d", "DE_Deere_ret_5d", "EMR_Emerson_ret_20d", "heston_var_ev_h5", "QQQ_vol_20d", "EQR_Equity_ret_1d", "SBUX_vol_20d"], "is_new": true}, {"model_id": "new_h3_GLOBAL_GradientBoosting_N15_t6", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 3, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "LLY_zscore_60d", "EOG_EOGResources_ret_5d", "SBUX_vol_20d", "vix_acceleration_1d", "DIS_vol_20d", "EWQ_France_zscore_60d", "BTI_BritishAmerican_ret_5d", "Core_PCE_zscore_60d", "GD_GeneralDynamics_zscore_60d", "EWL_Switzerland_zscore_60d", "HD_ret_5d", "MSTR_Bitcoin3_ret_5d", "EWM_Malaysia_zscore_60d"], "is_new": true}, {"model_id": "new_h3_GLOBAL_GradientBoosting_N15_t7", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 3, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "ORCL_zscore_60d", "LUV_SouthwestAir_ret_5d", "US7Y_Rate_ret_20d", "DE_Deere_ret_5d", "PLD_Prologis_ret_5d", "CTAS_Cintas_vol_20d", "spx_abs_ret_max_5d", "US3Y_Rate_ret_5d", "SLB_Schlumberger_ret_5d", "GILD_Gilead_ret_20d", "NEE_NextEra_ret_20d", "AMZN_ret_5d", "LMT_LockheedMartin_vol_20d"], "is_new": true}, {"model_id": "new_h3_GLOBAL_GradientBoosting_N20_t0", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 3, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "heston_var_ev_h3", "ORCL_zscore_60d", "HD_ret_5d", "BLK_BlackRock_zscore_60d", "XLV_Health_zscore_60d", "MSTR_Bitcoin3_ret_20d", "TM_Telephone_ret_1d", "EWM_Malaysia_zscore_60d", "EXC_Exelon_zscore_60d", "heston_var_ev_h7", "TGT_Target_zscore_60d", "ORCL_vol_20d", "XOM_ret_20d", "VVIX_ret_20d", "SLB_Schlumberger_ret_1d", "SBUX_zscore_60d", "CPB_CampbellSoup_ret_20d", "Nikkei_Japan_vol_20d"], "is_new": true}, {"model_id": "new_h3_GLOBAL_GradientBoosting_N20_t1", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 3, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "XLV_Health_zscore_60d", "DAX_Germany_zscore_60d", "US3M_Rate_vol_20d", "CMCSA_ret_1d", "SJM_JM_Smucker_ret_1d", "Retail_Sales_zscore_60d", "EWA_Australia_zscore_60d", "MS_MorganStanley_ret_1d", "AMGN_Amgen_ret_1d", "IWM_SmallCap_vol_20d", "EWQ_France_zscore_60d", "VVIX_ret_20d", "3M_ret_5d", "TXN_vol_20d", "BTI_BritishAmerican_ret_20d", "CI_Cigna_vol_20d", "AMD_ret_5d", "AMT_AmericanTower_ret_1d"], "is_new": true}, {"model_id": "new_h3_GLOBAL_GradientBoosting_N20_t2", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 3, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "Core_CPI_zscore_60d", "SPY_zscore_60d", "EFFR_vol_20d", "NEE_NextEra_ret_20d", "EWG_Germany_vol_20d", "Retail_Sales_zscore_60d", "heston_var_ev_h3", "EWY_Korea_zscore_60d", "Brent_Oil_FRED_ret_20d", "VOD_Vodafone_zscore_60d", "TM_Telephone_vol_20d", "SBUX_vol_20d", "AXP_Amex_vol_20d", "EWH_HongKong_ret_5d", "EQR_Equity_ret_1d", "ES_Evergy_ret_1d", "INTC_ret_5d", "BDX_Becton_Dickinson_ret_20d"], "is_new": true}, {"model_id": "new_h3_GLOBAL_GradientBoosting_N20_t3", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 3, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "XOM_ret_1d", "DOW_Price_zscore_60d", "EMR_Emerson_ret_20d", "QQQ_vol_20d", "EXC_Exelon_zscore_60d", "hmm_p_stress", "HD_zscore_60d", "MO_AltriaMG_ret_1d", "FedFunds_zscore_60d", "spx_momentum_3d", "LMT_LockheedMartin_ret_1d", "HD_ret_20d", "SBUX_ret_5d", "PAYX_Paychex_ret_20d", "SPY_zscore_60d", "SCHW_Schwab_ret_5d", "WTI_Oil_FRED_zscore_60d", "EOG_EOGResources_vol_20d"], "is_new": true}, {"model_id": "new_h3_GLOBAL_GradientBoosting_N20_t4", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 3, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "Brent_Oil_FRED_ret_5d", "PCAR_PaccarInc_ret_5d", "IBEX_Spain_ret_20d", "AMZN_ret_5d", "NEE_NextEra_ret_20d", "MS_MorganStanley_ret_5d", "US1Y_Rate_ret_5d", "MS_MorganStanley_ret_1d", "PAYX_Paychex_zscore_60d", "US3M_Rate_zscore_60d", "US5Y_Rate_ret_5d", "EXC_Exelon_zscore_60d", "DAX_Germany_zscore_60d", "3M_vol_20d", "MSTR_Bitcoin3_ret_1d", "Core_PCE_zscore_60d", "EWL_Switzerland_vol_20d", "PPL_PPL_ret_1d"], "is_new": true}, {"model_id": "new_h3_GLOBAL_GradientBoosting_N20_t5", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 3, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "MSTR_Bitcoin3_ret_1d", "CI_Cigna_vol_20d", "US30Y_Rate_ret_20d", "DE_Deere_ret_5d", "INTC_ret_5d", "XLB_Materials_zscore_60d", "EWG_Germany_vol_20d", "AORD_AUS_zscore_60d", "TXN_vol_20d", "SCHW_Schwab_ret_5d", "SPY_zscore_60d", "US3M_Rate_vol_20d", "WTI_Oil_FRED_zscore_60d", "BTI_BritishAmerican_ret_5d", "EQIX_Equinix_ret_5d", "SJM_JM_Smucker_ret_5d", "Brent_Oil_FRED_ret_5d", "EMR_Emerson_ret_20d"], "is_new": true}, {"model_id": "new_h3_GLOBAL_GradientBoosting_N20_t6", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 3, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "LUV_SouthwestAir_ret_5d", "ORCL_vol_20d", "Retail_Sales_zscore_60d", "Brent_Oil_FRED_ret_5d", "ASX_Australia_ret_5d", "EOG_EOGResources_vol_20d", "MS_MorganStanley_ret_1d", "EOG_EOGResources_ret_5d", "IWM_SmallCap_vol_20d", "CPB_CampbellSoup_vol_20d", "XLF_Fin_vol_20d", "EWG_Germany_ret_20d", "EWA_Australia_zscore_60d", "INTC_ret_5d", "NVDA_vol_20d", "DE_Deere_ret_5d", "SLB_Schlumberger_ret_1d", "HD_ret_5d"], "is_new": true}, {"model_id": "new_h3_GLOBAL_GradientBoosting_N20_t7", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 3, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "TM_Telephone_vol_20d", "ES_Evergy_ret_1d", "SJM_JM_Smucker_ret_5d", "EXC_Exelon_ret_1d", "AORD_AUS_zscore_60d", "CPB_CampbellSoup_ret_5d", "ENB_EnbridgeInc_ret_1d", "LMT_LockheedMartin_vol_20d", "DE_Deere_ret_5d", "BDX_Becton_Dickinson_ret_20d", "AXP_Amex_ret_20d", "FedFunds_zscore_60d", "EWS_Singapore_ret_5d", "US3M_Rate_vol_20d", "BTI_BritishAmerican_ret_5d", "EFFR_vol_20d", "hmm_p_stress", "Core_PCE_zscore_60d"], "is_new": true}, {"model_id": "new_h3_GLOBAL_GradientBoosting_N25_t0", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 3, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "spx_momentum_3d", "NVDA_vol_20d", "Nikkei_Japan_zscore_60d", "PLD_Prologis_ret_5d", "SO_SouthernCo_ret_5d", "EWG_Germany_vol_20d", "AMZN_ret_5d", "EWA_Australia_zscore_60d", "CPB_CampbellSoup_ret_20d", "SBUX_zscore_60d", "EOG_EOGResources_ret_5d", "VVIX_ret_20d", "GD_GeneralDynamics_zscore_60d", "AXP_Amex_vol_20d", "XLV_Health_zscore_60d", "XLK_Tech_zscore_60d", "EWL_Switzerland_vol_20d", "QQQ_vol_20d", "LMT_LockheedMartin_ret_1d", "CCI_CrownCastle_vol_20d", "EWJ_Japan_vol_20d", "SBUX_vol_20d", "NOC_Northrop_ret_20d"], "is_new": true}, {"model_id": "new_h3_GLOBAL_GradientBoosting_N25_t1", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 3, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "spx_abs_ret_max_5d", "CPB_CampbellSoup_vol_20d", "VVIX_ret_20d", "MSTR_Bitcoin3_ret_20d", "SCHW_Schwab_ret_5d", "US3M_Rate_vol_20d", "XLV_Health_zscore_60d", "CPB_CampbellSoup_ret_5d", "US1Y_Rate_ret_5d", "HD_zscore_60d", "IWM_SmallCap_vol_20d", "BTI_BritishAmerican_ret_5d", "XOM_ret_1d", "EFFR_ret_1d", "3M_ret_5d", "SLB_Schlumberger_ret_5d", "GE_ret_1d", "Nikkei_Japan_vol_20d", "LMT_LockheedMartin_ret_1d", "EWY_Korea_ret_20d", "SO_SouthernCo_ret_5d", "Industrial_Production_zscore_60d", "CMCSA_ret_1d"], "is_new": true}, {"model_id": "new_h3_GLOBAL_GradientBoosting_N25_t2", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 3, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "Michigan_Sentiment_ret_20d", "TM_Telephone_vol_20d", "HD_zscore_60d", "LLY_zscore_60d", "heston_ev_h3", "US6M_Rate_ret_20d", "EWJ_Japan_vol_20d", "EWY_Korea_zscore_60d", "BA_ret_1d", "CPB_CampbellSoup_ret_5d", "AXP_Amex_vol_20d", "PG_ret_20d", "LUV_SouthwestAir_ret_5d", "EOG_EOGResources_vol_20d", "T_ret_1d", "PAYX_Paychex_zscore_60d", "XLF_Fin_vol_20d", "QQQ_vol_20d", "ENB_EnbridgeInc_ret_1d", "DHR_ret_1d", "Industrial_Production_zscore_60d", "PPL_PPL_ret_1d", "ITT_ITTInc_ret_5d"], "is_new": true}, {"model_id": "new_h3_GLOBAL_GradientBoosting_N25_t3", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 3, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "XLV_Health_zscore_60d", "EWY_Korea_ret_20d", "Core_PCE_zscore_60d", "EFFR_vol_20d", "ES_Evergy_ret_1d", "QQQ_vol_20d", "EOG_EOGResources_ret_5d", "MRK_Merck_zscore_60d", "GD_GeneralDynamics_zscore_60d", "HD_ret_5d", "US3M_Rate_zscore_60d", "LUV_SouthwestAir_ret_5d", "INTC_ret_1d", "AMD_ret_5d", "vix_acceleration_1d", "XLY_Disc_vol_20d", "PAYX_Paychex_zscore_60d", "US5Y_Rate_ret_5d", "INTC_ret_5d", "TXN_vol_20d", "DE_Deere_ret_5d", "IYM_BasicMaterials_ret_20d", "EWY_Korea_zscore_60d"], "is_new": true}, {"model_id": "new_h3_GLOBAL_GradientBoosting_N25_t4", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 3, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWQ_France_ret_20d", "XLY_Disc_vol_20d", "MSTR_Bitcoin3_ret_1d", "PAYX_Paychex_ret_20d", "SBUX_vol_20d", "T10Y2Y_Spread_ret_5d", "EWA_Australia_ret_1d", "US3M_Rate_vol_20d", "AMD_ret_1d", "spx_momentum_3d", "INTC_ret_5d", "CCI_CrownCastle_vol_20d", "NEE_NextEra_ret_20d", "EWG_Germany_ret_20d", "TXN_vol_20d", "3M_vol_20d", "M_Macys_vol_20d", "Nikkei_Japan_vol_20d", "EQR_Equity_ret_1d", "Michigan_Sentiment_ret_20d", "SBUX_ret_5d", "HangSeng_HK_ret_1d", "AMD_ret_5d"], "is_new": true}, {"model_id": "new_h3_GLOBAL_GradientBoosting_N25_t5", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 3, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "PLD_Prologis_ret_5d", "DE_Deere_vol_20d", "EOG_EOGResources_ret_5d", "XLY_Disc_vol_20d", "CTAS_Cintas_vol_20d", "SJM_JM_Smucker_ret_1d", "MS_MorganStanley_ret_1d", "GILD_Gilead_ret_20d", "TGT_Target_zscore_60d", "AMD_ret_5d", "EWL_Switzerland_vol_20d", "M_Macys_vol_20d", "BTI_BritishAmerican_ret_5d", "HD_ret_5d", "LOW_Lowes_ret_20d", "HangSeng_HK_ret_5d", "EWY_Korea_zscore_60d", "AMZN_ret_5d", "EQR_Equity_ret_1d", "VVIX_ret_20d", "CCI_CrownCastle_vol_20d", "LOW_Lowes_ret_5d", "MSTR_Bitcoin3_ret_20d"], "is_new": true}, {"model_id": "new_h3_GLOBAL_GradientBoosting_N25_t6", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 3, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "US30Y_Rate_ret_20d", "HangSeng_HK_ret_1d", "Brent_Oil_FRED_ret_20d", "heston_var_ev_h3", "GE_ret_1d", "US3Y_Rate_ret_5d", "EQIX_Equinix_ret_5d", "DAX_Germany_zscore_60d", "TM_Telephone_ret_1d", "EWM_Malaysia_ret_1d", "IWM_SmallCap_vol_20d", "M_Macys_vol_20d", "ASX_Australia_ret_5d", "DE_Deere_vol_20d", "XLY_Disc_vol_20d", "VRP_ma5", "EWM_Malaysia_zscore_60d", "LOW_Lowes_ret_20d", "BTI_BritishAmerican_ret_20d", "CCI_CrownCastle_vol_20d", "SBUX_vol_20d", "vix_acceleration_1d", "SLB_Schlumberger_ret_5d"], "is_new": true}, {"model_id": "new_h3_GLOBAL_GradientBoosting_N25_t7", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 3, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "XOM_ret_1d", "EWS_Singapore_ret_5d", "TM_Telephone_vol_20d", "AXP_Amex_ret_20d", "LMT_LockheedMartin_ret_1d", "TM_Telephone_ret_1d", "EWY_Korea_zscore_60d", "MS_MorganStanley_ret_5d", "ASX_Australia_ret_5d", "QQQ_vol_20d", "3M_vol_20d", "FedFunds_zscore_60d", "EWM_Malaysia_vol_20d", "Industrial_Production_zscore_60d", "CPB_CampbellSoup_vol_20d", "SBUX_vol_20d", "MRK_Merck_zscore_60d", "EWA_Australia_zscore_60d", "CCI_CrownCastle_vol_20d", "ORCL_zscore_60d", "EMR_Emerson_ret_20d", "NWL_Newell_ret_20d", "BTI_BritishAmerican_ret_20d"], "is_new": true}, {"model_id": "new_h3_GLOBAL_GradientBoosting_N30_t0", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 3, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWG_Germany_ret_20d", "heston_var_ev_h7", "Brent_Oil_FRED_ret_5d", "EWY_Korea_zscore_60d", "PAYX_Paychex_vol_20d", "VRP_ma5", "MS_MorganStanley_zscore_60d", "CPB_CampbellSoup_vol_20d", "EQIX_Equinix_ret_5d", "NVDA_vol_20d", "IYR_US_REIT2_zscore_60d", "AXP_Amex_vol_20d", "TED_Spread_zscore_60d", "TM_Telephone_ret_1d", "XLF_Fin_vol_20d", "XLB_Materials_zscore_60d", "SBUX_vol_20d", "LOW_Lowes_ret_5d", "gjr_condvar_h1", "SLB_Schlumberger_ret_1d", "INTC_ret_5d", "EWA_Australia_zscore_60d", "QQQ_vol_20d", "Retail_Sales_zscore_60d", "XLY_Disc_vol_20d", "SLB_Schlumberger_ret_5d", "CCI_CrownCastle_vol_20d", "Nikkei_Japan_vol_20d"], "is_new": true}, {"model_id": "new_h3_GLOBAL_GradientBoosting_N30_t1", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 3, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "IYM_BasicMaterials_ret_20d", "Brent_Oil_FRED_ret_5d", "CPB_CampbellSoup_vol_20d", "US5Y_Rate_ret_5d", "T_ret_1d", "SLB_Schlumberger_ret_1d", "US1Y_Rate_ret_20d", "PAYX_Paychex_vol_20d", "HD_ret_5d", "HangSeng_HK_ret_5d", "QQQ_vol_20d", "EWA_Australia_zscore_60d", "EQR_Equity_ret_1d", "CI_Cigna_vol_20d", "gjr_condvar_h1", "PAYX_Paychex_zscore_60d", "SLB_Schlumberger_ret_5d", "ITT_ITTInc_ret_5d", "HUM_Humana_ret_5d", "INTC_ret_5d", "PG_ret_20d", "heston_var_ev_h3", "Retail_Sales_zscore_60d", "VRP_ma5", "AMT_AmericanTower_ret_1d", "MS_MorganStanley_zscore_60d", "AVB_AvalonBay_zscore_60d", "T10Y2Y_Spread_ret_5d"], "is_new": true}, {"model_id": "new_h3_GLOBAL_GradientBoosting_N30_t2", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 3, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "IBEX_Spain_ret_20d", "TM_Telephone_vol_20d", "MRK_Merck_zscore_60d", "NOC_Northrop_ret_20d", "HD_ret_20d", "EWM_Malaysia_zscore_60d", "ENB_EnbridgeInc_ret_1d", "PLD_Prologis_ret_5d", "LUV_SouthwestAir_ret_5d", "HangSeng_HK_vol_20d", "CI_Cigna_vol_20d", "Brent_Oil_FRED_ret_20d", "ITT_ITTInc_ret_5d", "Nikkei_Japan_vol_20d", "heston_var_ev_h3", "DOW_Price_zscore_60d", "QQQ_vol_20d", "PCAR_PaccarInc_ret_5d", "BA_ret_1d", "gjr_condvar_h1", "LLY_zscore_60d", "MSTR_Bitcoin3_ret_5d", "EMR_Emerson_ret_20d", "MS_MorganStanley_ret_1d", "M_Macys_vol_20d", "US3M_Rate_vol_20d", "TM_Telephone_ret_1d", "CCI_CrownCastle_vol_20d"], "is_new": true}, {"model_id": "new_h3_GLOBAL_GradientBoosting_N30_t3", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 3, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "PAYX_Paychex_zscore_60d", "VOD_Vodafone_zscore_60d", "DOW_Price_zscore_60d", "AMZN_ret_5d", "AXP_Amex_ret_20d", "US1Y_Rate_ret_5d", "IWM_SmallCap_vol_20d", "EXC_Exelon_ret_1d", "MS_MorganStanley_zscore_60d", "EWQ_France_ret_20d", "CPB_CampbellSoup_vol_20d", "NWL_Newell_ret_20d", "HD_ret_1d", "NEE_NextEra_ret_20d", "AVB_AvalonBay_zscore_60d", "M_Macys_vol_20d", "XLK_Tech_zscore_60d", "IBEX_Spain_ret_20d", "US3M_Rate_vol_20d", "EWG_Germany_ret_20d", "EMR_Emerson_ret_20d", "EWQ_France_zscore_60d", "SBUX_vol_20d", "IYR_US_REIT2_zscore_60d", "IYM_BasicMaterials_ret_20d", "TM_Telephone_ret_1d", "EWH_HongKong_ret_5d", "SO_SouthernCo_ret_5d"], "is_new": true}, {"model_id": "new_h3_GLOBAL_GradientBoosting_N30_t4", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 3, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "spx_abs_ret_max_5d", "CLX_Clorox_vol_20d", "FedFunds_zscore_60d", "US6M_Rate_ret_20d", "EWY_Korea_ret_20d", "hmm_p_stress", "EFFR_ret_1d", "LOW_Lowes_ret_20d", "HD_ret_5d", "SLB_Schlumberger_ret_5d", "IWM_SmallCap_vol_20d", "US5Y_Rate_ret_5d", "ENB_EnbridgeInc_ret_1d", "SJM_JM_Smucker_ret_5d", "EOG_EOGResources_vol_20d", "AMD_ret_1d", "MS_MorganStanley_ret_5d", "Nikkei_Japan_vol_20d", "ORCL_vol_20d", "ITT_ITTInc_ret_5d", "EWH_HongKong_ret_5d", "EXC_Exelon_zscore_60d", "SJM_JM_Smucker_ret_1d", "PAYX_Paychex_vol_20d", "vix_acceleration_1d", "ASX_Australia_ret_5d", "US3M_Rate_zscore_60d", "AMT_AmericanTower_ret_1d"], "is_new": true}, {"model_id": "new_h3_GLOBAL_GradientBoosting_N30_t5", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 3, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "AVB_AvalonBay_zscore_60d", "AMT_AmericanTower_ret_1d", "EWG_Germany_vol_20d", "ENB_EnbridgeInc_ret_1d", "NVDA_vol_20d", "M_Macys_vol_20d", "NEE_NextEra_ret_20d", "CMCSA_ret_1d", "PLD_Prologis_ret_5d", "US30Y_Rate_ret_20d", "AMD_ret_5d", "SLB_Schlumberger_ret_5d", "XOM_ret_20d", "Michigan_Sentiment_ret_20d", "EXC_Exelon_ret_1d", "HangSeng_HK_vol_20d", "US6M_Rate_ret_20d", "BTI_BritishAmerican_ret_5d", "MS_MorganStanley_ret_5d", "DHR_vol_20d", "SCHW_Schwab_ret_5d", "SBUX_vol_20d", "NFCI_ret_5d", "LLY_zscore_60d", "3M_ret_5d", "DIS_vol_20d", "HUM_Humana_ret_5d", "EMR_Emerson_ret_20d"], "is_new": true}, {"model_id": "new_h3_GLOBAL_GradientBoosting_N30_t6", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 3, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "DE_Deere_vol_20d", "US3Y_Rate_ret_5d", "Core_CPI_zscore_60d", "LOW_Lowes_ret_5d", "EWG_Germany_ret_20d", "MO_AltriaMG_ret_1d", "spx_abs_ret_max_5d", "PCAR_PaccarInc_ret_5d", "IWM_SmallCap_vol_20d", "US7Y_Rate_ret_20d", "EFFR_vol_20d", "SBUX_ret_5d", "Brent_Oil_FRED_ret_20d", "VRP_ma5", "PG_ret_20d", "MS_MorganStanley_zscore_60d", "MS_MorganStanley_ret_5d", "spx_vol_5d", "MSTR_Bitcoin3_ret_1d", "ENB_EnbridgeInc_ret_1d", "HD_ret_5d", "vix_mean_abs_ret_5d", "EWA_Australia_ret_1d", "EWJ_Japan_vol_20d", "TED_Spread_zscore_60d", "AMZN_ret_5d", "CTAS_Cintas_vol_20d", "AXP_Amex_vol_20d"], "is_new": true}, {"model_id": "new_h3_GLOBAL_GradientBoosting_N30_t7", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 3, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "PG_ret_20d", "HUM_Humana_ret_5d", "HangSeng_HK_vol_20d", "MS_MorganStanley_zscore_60d", "AMT_AmericanTower_ret_1d", "TM_Telephone_vol_20d", "EWL_Switzerland_zscore_60d", "vix_acceleration_1d", "US30Y_Rate_ret_20d", "MSTR_Bitcoin3_ret_1d", "FedFunds_zscore_60d", "US5Y_Rate_ret_5d", "Core_CPI_zscore_60d", "SBUX_vol_20d", "ITT_ITTInc_ret_5d", "AMD_ret_1d", "PCAR_PaccarInc_ret_5d", "T_ret_1d", "EWG_Germany_ret_20d", "CPB_CampbellSoup_ret_20d", "DE_Deere_ret_5d", "EWG_Germany_vol_20d", "heston_var_ev_h7", "XOM_ret_1d", "INTC_ret_5d", "EWM_Malaysia_ret_1d", "GE_ret_1d", "SJM_JM_Smucker_ret_1d"], "is_new": true}, {"model_id": "new_h3_GLOBAL_RandomForest_N5_t0", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 3, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "HD_ret_5d", "Brent_Oil_FRED_ret_20d", "heston_var_ev_h3"], "is_new": true}, {"model_id": "new_h3_GLOBAL_RandomForest_N5_t1", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 3, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "CPB_CampbellSoup_vol_20d", "HangSeng_HK_ret_1d", "CPB_CampbellSoup_zscore_60d"], "is_new": true}, {"model_id": "new_h3_GLOBAL_RandomForest_N5_t2", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 3, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "TED_Spread_zscore_60d", "US5Y_Rate_ret_5d", "US30Y_Rate_ret_20d"], "is_new": true}, {"model_id": "new_h3_GLOBAL_RandomForest_N5_t3", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 3, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "heston_var_ev_h7", "DE_Deere_ret_5d", "PAYX_Paychex_ret_20d"], "is_new": true}, {"model_id": "new_h3_GLOBAL_RandomForest_N5_t4", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 3, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "TED_Spread_vol_20d", "HD_zscore_60d", "XOM_ret_1d"], "is_new": true}, {"model_id": "new_h3_GLOBAL_RandomForest_N5_t5", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 3, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "LOW_Lowes_ret_20d", "spx_abs_ret_max_5d", "vix_acceleration_1d"], "is_new": true}, {"model_id": "new_h3_GLOBAL_RandomForest_N5_t6", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 3, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "spx_momentum_3d", "WTI_Oil_FRED_zscore_60d", "vix_mean_abs_ret_5d"], "is_new": true}, {"model_id": "new_h3_GLOBAL_RandomForest_N5_t7", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 3, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "DE_Deere_ret_5d", "ITT_ITTInc_ret_5d", "EWA_Australia_ret_1d"], "is_new": true}, {"model_id": "new_h3_GLOBAL_RandomForest_N8_t0", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 3, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "CI_Cigna_vol_20d", "US3Y_Rate_ret_5d", "MS_MorganStanley_ret_1d", "LUV_SouthwestAir_ret_5d", "GE_ret_1d", "US3M_Rate_zscore_60d"], "is_new": true}, {"model_id": "new_h3_GLOBAL_RandomForest_N8_t1", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 3, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "XOM_ret_1d", "HD_zscore_60d", "EXC_Exelon_ret_1d", "LOW_Lowes_ret_5d", "spx_vol_5d", "VOD_Vodafone_zscore_60d"], "is_new": true}, {"model_id": "new_h3_GLOBAL_RandomForest_N8_t2", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 3, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "DIS_vol_20d", "Brent_Oil_FRED_ret_20d", "3M_vol_20d", "DHR_ret_1d", "ASX_Australia_vol_20d", "heston_var_ev_h3"], "is_new": true}, {"model_id": "new_h3_GLOBAL_RandomForest_N8_t3", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 3, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "Brent_Oil_FRED_ret_20d", "EWY_Korea_ret_20d", "heston_ev_h3", "Nikkei_Japan_vol_20d", "AMT_AmericanTower_ret_1d", "HD_ret_5d"], "is_new": true}, {"model_id": "new_h3_GLOBAL_RandomForest_N8_t4", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 3, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "3M_vol_20d", "ASX_Australia_ret_5d", "HD_ret_5d", "NEE_NextEra_ret_20d", "TM_Telephone_vol_20d", "SBUX_vol_20d"], "is_new": true}, {"model_id": "new_h3_GLOBAL_RandomForest_N8_t5", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 3, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "CCI_CrownCastle_vol_20d", "AVB_AvalonBay_zscore_60d", "BLK_BlackRock_zscore_60d", "TED_Spread_vol_20d", "HD_ret_20d", "US1Y_Rate_ret_20d"], "is_new": true}, {"model_id": "new_h3_GLOBAL_RandomForest_N8_t6", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 3, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "LLY_zscore_60d", "IYM_BasicMaterials_ret_20d", "Nikkei_Japan_zscore_60d", "SPY_zscore_60d", "MSTR_Bitcoin3_ret_5d", "DHR_vol_20d"], "is_new": true}, {"model_id": "new_h3_GLOBAL_RandomForest_N8_t7", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 3, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "GD_GeneralDynamics_zscore_60d", "HD_ret_20d", "CPB_CampbellSoup_ret_20d", "PFE_ret_1d", "CLX_Clorox_vol_20d", "AMGN_Amgen_ret_1d"], "is_new": true}, {"model_id": "new_h3_GLOBAL_RandomForest_N10_t0", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 3, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "WTI_Oil_FRED_zscore_60d", "DHR_vol_20d", "XLY_Disc_vol_20d", "EWL_Switzerland_vol_20d", "heston_var_ev_h7", "NFCI_ret_5d", "US3Y_Rate_ret_5d", "ASX_Australia_ret_5d"], "is_new": true}, {"model_id": "new_h3_GLOBAL_RandomForest_N10_t1", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 3, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "SPY_zscore_60d", "heston_var_ev_h7", "SCHW_Schwab_ret_5d", "DHR_vol_20d", "EWQ_France_zscore_60d", "IYR_US_REIT2_zscore_60d", "TED_Spread_vol_20d", "EWA_Australia_ret_1d"], "is_new": true}, {"model_id": "new_h3_GLOBAL_RandomForest_N10_t2", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 3, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "LUV_SouthwestAir_ret_5d", "heston_ev_h3", "DHR_ret_1d", "BA_ret_1d", "HangSeng_HK_ret_5d", "PG_ret_20d", "SBUX_ret_5d", "AMT_AmericanTower_ret_1d"], "is_new": true}, {"model_id": "new_h3_GLOBAL_RandomForest_N10_t3", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 3, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "PAYX_Paychex_vol_20d", "MS_MorganStanley_ret_5d", "DE_Deere_ret_5d", "PG_ret_20d", "EWY_Korea_ret_20d", "EOG_EOGResources_ret_5d", "LOW_Lowes_ret_5d", "JNJ_ret_1d"], "is_new": true}, {"model_id": "new_h3_GLOBAL_RandomForest_N10_t4", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 3, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWL_Switzerland_vol_20d", "US1Y_Rate_ret_5d", "EWM_Malaysia_zscore_60d", "EFFR_ret_1d", "MS_MorganStanley_zscore_60d", "EMR_Emerson_ret_20d", "DAX_Germany_zscore_60d", "Industrial_Production_zscore_60d"], "is_new": true}, {"model_id": "new_h3_GLOBAL_RandomForest_N10_t5", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 3, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "TED_Spread_zscore_60d", "EWH_HongKong_ret_5d", "PCAR_PaccarInc_ret_5d", "EWQ_France_ret_20d", "DHR_ret_1d", "EWM_Malaysia_zscore_60d", "3M_ret_5d", "Brent_Oil_FRED_ret_20d"], "is_new": true}, {"model_id": "new_h3_GLOBAL_RandomForest_N10_t6", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 3, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "SBUX_ret_5d", "LUV_SouthwestAir_ret_5d", "AORD_AUS_zscore_60d", "EWY_Korea_ret_20d", "TED_Spread_zscore_60d", "BDX_Becton_Dickinson_ret_20d", "XLB_Materials_zscore_60d", "IWM_SmallCap_vol_20d"], "is_new": true}, {"model_id": "new_h3_GLOBAL_RandomForest_N10_t7", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 3, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "ENB_EnbridgeInc_ret_1d", "BLK_BlackRock_zscore_60d", "EWJ_Japan_vol_20d", "US7Y_Rate_ret_20d", "SCHW_Schwab_ret_5d", "CPB_CampbellSoup_vol_20d", "AXP_Amex_ret_20d", "Core_PCE_zscore_60d"], "is_new": true}, {"model_id": "new_h3_GLOBAL_RandomForest_N12_t0", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 3, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWL_Switzerland_vol_20d", "ASX_Australia_vol_20d", "LUV_SouthwestAir_ret_5d", "AXP_Amex_ret_20d", "US7Y_Rate_ret_20d", "TM_Telephone_vol_20d", "BA_ret_1d", "US6M_Rate_ret_20d", "VOD_Vodafone_zscore_60d", "EWQ_France_zscore_60d"], "is_new": true}, {"model_id": "new_h3_GLOBAL_RandomForest_N12_t1", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 3, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "BDX_Becton_Dickinson_ret_20d", "TED_Spread_zscore_60d", "SBUX_ret_5d", "Michigan_Sentiment_ret_20d", "SJM_JM_Smucker_ret_5d", "HD_zscore_60d", "vix_acceleration_1d", "TED_Spread_vol_20d", "MS_MorganStanley_ret_5d", "MRK_Merck_zscore_60d"], "is_new": true}, {"model_id": "new_h3_GLOBAL_RandomForest_N12_t2", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 3, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "LMT_LockheedMartin_ret_1d", "ES_Evergy_ret_1d", "PG_ret_20d", "ORCL_zscore_60d", "MS_MorganStanley_ret_1d", "T10Y2Y_Spread_ret_5d", "LMT_LockheedMartin_vol_20d", "PLD_Prologis_ret_5d", "IYR_US_REIT2_zscore_60d", "LUV_SouthwestAir_ret_5d"], "is_new": true}, {"model_id": "new_h3_GLOBAL_RandomForest_N12_t3", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 3, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "DHR_vol_20d", "US6M_Rate_ret_20d", "EOG_EOGResources_vol_20d", "SCHW_Schwab_ret_5d", "JNJ_ret_1d", "AMT_AmericanTower_ret_1d", "CLX_Clorox_vol_20d", "XLB_Materials_zscore_60d", "EWH_HongKong_ret_5d", "DIS_vol_20d"], "is_new": true}, {"model_id": "new_h3_GLOBAL_RandomForest_N12_t4", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 3, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "IBEX_Spain_ret_20d", "XLK_Tech_zscore_60d", "HD_ret_1d", "DHR_ret_1d", "ES_Evergy_ret_1d", "EOG_EOGResources_vol_20d", "CMCSA_ret_1d", "NWL_Newell_ret_20d", "MSTR_Bitcoin3_ret_5d", "BDX_Becton_Dickinson_ret_20d"], "is_new": true}, {"model_id": "new_h3_GLOBAL_RandomForest_N12_t5", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 3, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "SLB_Schlumberger_ret_5d", "CPB_CampbellSoup_vol_20d", "GILD_Gilead_ret_20d", "XLF_Fin_vol_20d", "T_ret_1d", "PAYX_Paychex_zscore_60d", "Brent_Oil_FRED_ret_5d", "HD_ret_5d", "EQR_Equity_ret_1d", "AMT_AmericanTower_ret_1d"], "is_new": true}, {"model_id": "new_h3_GLOBAL_RandomForest_N12_t6", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 3, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "SJM_JM_Smucker_ret_1d", "PAYX_Paychex_zscore_60d", "XOM_ret_1d", "EWL_Switzerland_zscore_60d", "Core_PCE_zscore_60d", "EQIX_Equinix_ret_5d", "EWY_Korea_zscore_60d", "AXP_Amex_vol_20d", "HD_zscore_60d", "VOD_Vodafone_zscore_60d"], "is_new": true}, {"model_id": "new_h3_GLOBAL_RandomForest_N12_t7", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 3, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "DIS_vol_20d", "EWM_Malaysia_vol_20d", "BTI_BritishAmerican_ret_5d", "SBUX_zscore_60d", "Industrial_Production_zscore_60d", "heston_var_ev_h3", "LMT_LockheedMartin_ret_1d", "EWG_Germany_vol_20d", "EWL_Switzerland_vol_20d", "PLD_Prologis_ret_5d"], "is_new": true}, {"model_id": "new_h3_GLOBAL_RandomForest_N15_t0", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 3, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "hmm_p_stress", "Nikkei_Japan_zscore_60d", "EWL_Switzerland_vol_20d", "BLK_BlackRock_zscore_60d", "ORCL_vol_20d", "INTC_ret_5d", "TM_Telephone_ret_1d", "LOW_Lowes_ret_20d", "SLB_Schlumberger_ret_5d", "DHR_vol_20d", "PCAR_PaccarInc_ret_5d", "HangSeng_HK_ret_1d", "gjr_condvar_h1"], "is_new": true}, {"model_id": "new_h3_GLOBAL_RandomForest_N15_t1", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 3, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWL_Switzerland_zscore_60d", "heston_ev_h3", "AORD_AUS_zscore_60d", "heston_var_ev_h3", "EWS_Singapore_ret_5d", "SBUX_zscore_60d", "SLB_Schlumberger_ret_5d", "ITT_ITTInc_ret_5d", "TED_Spread_zscore_60d", "CPB_CampbellSoup_vol_20d", "GD_GeneralDynamics_zscore_60d", "vix_mean_abs_ret_5d", "heston_var_ev_h5"], "is_new": true}, {"model_id": "new_h3_GLOBAL_RandomForest_N15_t2", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 3, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "TGT_Target_zscore_60d", "PLD_Prologis_ret_5d", "NEE_NextEra_ret_20d", "Core_CPI_zscore_60d", "XOM_ret_1d", "AMT_AmericanTower_ret_1d", "PFE_ret_1d", "INTC_ret_1d", "AMZN_ret_5d", "INTC_ret_5d", "MS_MorganStanley_zscore_60d", "CCI_CrownCastle_vol_20d", "CPB_CampbellSoup_ret_5d"], "is_new": true}, {"model_id": "new_h3_GLOBAL_RandomForest_N15_t3", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 3, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "CPB_CampbellSoup_ret_20d", "AXP_Amex_ret_20d", "SCHW_Schwab_ret_5d", "HangSeng_HK_ret_1d", "VVIX_ret_20d", "XOM_ret_20d", "TXN_vol_20d", "DAX_Germany_vol_20d", "US30Y_Rate_ret_20d", "T10Y2Y_Spread_ret_5d", "M_Macys_vol_20d", "XLF_Fin_vol_20d", "MSTR_Bitcoin3_ret_20d"], "is_new": true}, {"model_id": "new_h3_GLOBAL_RandomForest_N15_t4", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 3, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "BTI_BritishAmerican_ret_20d", "SBUX_vol_20d", "Brent_Oil_FRED_ret_20d", "SJM_JM_Smucker_ret_5d", "ORCL_zscore_60d", "DAX_Germany_zscore_60d", "VRP_ma5", "PCAR_PaccarInc_ret_5d", "AMD_ret_1d", "CCI_CrownCastle_vol_20d", "ES_Evergy_ret_1d", "NOC_Northrop_ret_20d", "SLB_Schlumberger_ret_1d"], "is_new": true}, {"model_id": "new_h3_GLOBAL_RandomForest_N15_t5", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 3, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EOG_EOGResources_ret_5d", "heston_var_ev_h7", "EXC_Exelon_zscore_60d", "ES_Evergy_ret_1d", "BLK_BlackRock_zscore_60d", "MS_MorganStanley_ret_1d", "AXP_Amex_vol_20d", "MRK_Merck_zscore_60d", "heston_var_ev_h3", "BTI_BritishAmerican_ret_5d", "CPB_CampbellSoup_ret_20d", "Michigan_Sentiment_ret_20d", "XOM_ret_1d"], "is_new": true}, {"model_id": "new_h3_GLOBAL_RandomForest_N15_t6", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 3, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "GD_GeneralDynamics_zscore_60d", "EWY_Korea_ret_20d", "IYR_US_REIT2_zscore_60d", "DOW_Price_zscore_60d", "Michigan_Sentiment_ret_20d", "US1Y_Rate_ret_20d", "DAX_Germany_zscore_60d", "BTI_BritishAmerican_ret_5d", "XLB_Materials_zscore_60d", "HangSeng_HK_ret_1d", "PAYX_Paychex_vol_20d", "CI_Cigna_vol_20d", "US6M_Rate_ret_20d"], "is_new": true}, {"model_id": "new_h3_GLOBAL_RandomForest_N15_t7", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 3, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EOG_EOGResources_vol_20d", "SJM_JM_Smucker_ret_1d", "CPB_CampbellSoup_vol_20d", "ASX_Australia_ret_5d", "QQQ_vol_20d", "EWY_Korea_ret_20d", "DAX_Germany_zscore_60d", "JNJ_ret_1d", "PAYX_Paychex_zscore_60d", "LUV_SouthwestAir_ret_5d", "XOM_ret_1d", "DHR_ret_1d", "Core_PCE_zscore_60d"], "is_new": true}, {"model_id": "new_h3_GLOBAL_RandomForest_N20_t0", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 3, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "HangSeng_HK_ret_1d", "SLB_Schlumberger_ret_5d", "XLK_Tech_zscore_60d", "US3M_Rate_vol_20d", "heston_ev_h3", "BLK_BlackRock_zscore_60d", "IWM_SmallCap_vol_20d", "NEE_NextEra_ret_20d", "EWY_Korea_zscore_60d", "EWG_Germany_vol_20d", "CCI_CrownCastle_vol_20d", "PAYX_Paychex_ret_20d", "DHR_ret_1d", "BDX_Becton_Dickinson_ret_20d", "heston_var_ev_h7", "spx_abs_ret_max_5d", "HD_ret_1d", "MS_MorganStanley_zscore_60d"], "is_new": true}, {"model_id": "new_h3_GLOBAL_RandomForest_N20_t1", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 3, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "PCAR_PaccarInc_ret_5d", "AMGN_Amgen_ret_1d", "hmm_p_stress", "BTI_BritishAmerican_ret_5d", "US3M_Rate_vol_20d", "TM_Telephone_ret_1d", "SCHW_Schwab_ret_5d", "TGT_Target_zscore_60d", "EWH_HongKong_ret_5d", "LMT_LockheedMartin_ret_1d", "AMT_AmericanTower_ret_1d", "INTC_ret_5d", "HangSeng_HK_ret_5d", "HUM_Humana_ret_5d", "LOW_Lowes_ret_5d", "XLY_Disc_vol_20d", "EWA_Australia_ret_1d", "CPB_CampbellSoup_zscore_60d"], "is_new": true}, {"model_id": "new_h3_GLOBAL_RandomForest_N20_t2", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 3, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "DOW_Price_zscore_60d", "LOW_Lowes_ret_20d", "HD_zscore_60d", "PAYX_Paychex_ret_20d", "3M_ret_5d", "US5Y_Rate_ret_5d", "US6M_Rate_ret_20d", "SJM_JM_Smucker_ret_1d", "EWA_Australia_zscore_60d", "EXC_Exelon_ret_1d", "CPB_CampbellSoup_vol_20d", "EWQ_France_ret_20d", "hmm_p_stress", "AMT_AmericanTower_ret_1d", "JNJ_ret_1d", "HD_ret_1d", "3M_vol_20d", "PAYX_Paychex_zscore_60d"], "is_new": true}, {"model_id": "new_h3_GLOBAL_RandomForest_N20_t3", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 3, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "hmm_p_stress", "US1Y_Rate_ret_5d", "MSTR_Bitcoin3_ret_5d", "DE_Deere_ret_5d", "gjr_condvar_h1", "VOD_Vodafone_zscore_60d", "GE_ret_1d", "EWA_Australia_zscore_60d", "NEE_NextEra_ret_20d", "T_ret_1d", "CTAS_Cintas_vol_20d", "EWA_Australia_ret_1d", "CLX_Clorox_vol_20d", "HUM_Humana_ret_5d", "IYR_US_REIT2_zscore_60d", "EWM_Malaysia_zscore_60d", "XLV_Health_zscore_60d", "HD_ret_20d"], "is_new": true}, {"model_id": "new_h3_GLOBAL_RandomForest_N20_t4", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 3, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "AMT_AmericanTower_ret_1d", "ORCL_zscore_60d", "TGT_Target_zscore_60d", "EWY_Korea_zscore_60d", "LLY_zscore_60d", "vix_acceleration_1d", "SPY_zscore_60d", "GILD_Gilead_ret_20d", "IYM_BasicMaterials_ret_20d", "TM_Telephone_ret_1d", "HangSeng_HK_ret_5d", "BA_ret_1d", "EQIX_Equinix_ret_5d", "DE_Deere_ret_5d", "EMR_Emerson_ret_20d", "PAYX_Paychex_ret_20d", "SBUX_ret_5d", "MSTR_Bitcoin3_ret_1d"], "is_new": true}, {"model_id": "new_h3_GLOBAL_RandomForest_N20_t5", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 3, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "US1Y_Rate_ret_20d", "BA_ret_1d", "NWL_Newell_ret_20d", "ORCL_vol_20d", "SBUX_vol_20d", "DAX_Germany_zscore_60d", "IYR_US_REIT2_zscore_60d", "EQIX_Equinix_ret_5d", "PAYX_Paychex_vol_20d", "VRP_ma5", "EWA_Australia_zscore_60d", "HD_ret_20d", "EXC_Exelon_ret_1d", "GE_ret_1d", "LMT_LockheedMartin_ret_1d", "DAX_Germany_vol_20d", "VOD_Vodafone_zscore_60d", "ASX_Australia_vol_20d"], "is_new": true}, {"model_id": "new_h3_GLOBAL_RandomForest_N20_t6", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 3, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "heston_var_ev_h5", "DHR_vol_20d", "BTI_BritishAmerican_ret_20d", "3M_vol_20d", "US1Y_Rate_ret_5d", "EWA_Australia_zscore_60d", "AMD_ret_5d", "NOC_Northrop_ret_20d", "EWH_HongKong_ret_5d", "GILD_Gilead_ret_20d", "HUM_Humana_ret_5d", "EWG_Germany_ret_20d", "SBUX_zscore_60d", "CPB_CampbellSoup_vol_20d", "AMGN_Amgen_ret_1d", "vix_mean_abs_ret_5d", "SPY_zscore_60d", "EXC_Exelon_ret_1d"], "is_new": true}, {"model_id": "new_h3_GLOBAL_RandomForest_N20_t7", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 3, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "GE_ret_1d", "ES_Evergy_ret_1d", "EWM_Malaysia_vol_20d", "PFE_ret_1d", "EWQ_France_zscore_60d", "Industrial_Production_zscore_60d", "EWH_HongKong_ret_5d", "EXC_Exelon_zscore_60d", "US3M_Rate_vol_20d", "PCAR_PaccarInc_ret_5d", "DAX_Germany_zscore_60d", "IBEX_Spain_ret_20d", "T10Y2Y_Spread_ret_5d", "NVDA_vol_20d", "XLY_Disc_vol_20d", "SLB_Schlumberger_ret_5d", "hmm_p_stress", "EWA_Australia_zscore_60d"], "is_new": true}, {"model_id": "new_h3_GLOBAL_RandomForest_N25_t0", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 3, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "Industrial_Production_zscore_60d", "AMZN_ret_5d", "ES_Evergy_ret_1d", "EQR_Equity_ret_1d", "PAYX_Paychex_vol_20d", "spx_abs_ret_max_5d", "EXC_Exelon_zscore_60d", "MS_MorganStanley_zscore_60d", "heston_var_ev_h3", "US6M_Rate_ret_20d", "3M_vol_20d", "EWA_Australia_zscore_60d", "EWM_Malaysia_vol_20d", "SPY_zscore_60d", "EXC_Exelon_ret_1d", "SBUX_zscore_60d", "CTAS_Cintas_vol_20d", "EWL_Switzerland_zscore_60d", "Retail_Sales_zscore_60d", "AMD_ret_1d", "vix_mean_abs_ret_5d", "TED_Spread_zscore_60d", "SCHW_Schwab_ret_5d"], "is_new": true}, {"model_id": "new_h3_GLOBAL_RandomForest_N25_t1", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 3, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EMR_Emerson_ret_20d", "EQR_Equity_ret_1d", "CI_Cigna_vol_20d", "XOM_ret_20d", "vix_mean_abs_ret_5d", "AXP_Amex_ret_20d", "ASX_Australia_ret_5d", "HangSeng_HK_vol_20d", "AMD_ret_5d", "heston_ev_h3", "DOW_Price_zscore_60d", "EWJ_Japan_vol_20d", "HD_ret_1d", "DE_Deere_vol_20d", "AXP_Amex_vol_20d", "EWA_Australia_ret_1d", "EWM_Malaysia_ret_1d", "TXN_vol_20d", "Retail_Sales_zscore_60d", "EWH_HongKong_ret_5d", "US1Y_Rate_ret_20d", "HangSeng_HK_ret_5d", "MRK_Merck_zscore_60d"], "is_new": true}, {"model_id": "new_h3_GLOBAL_RandomForest_N25_t2", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 3, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "DAX_Germany_zscore_60d", "CPB_CampbellSoup_ret_20d", "INTC_ret_1d", "NFCI_ret_5d", "EWQ_France_zscore_60d", "TXN_vol_20d", "EWC_Canada_zscore_60d", "Michigan_Sentiment_ret_20d", "MSTR_Bitcoin3_ret_1d", "NOC_Northrop_ret_20d", "US3M_Rate_vol_20d", "HD_ret_1d", "EWA_Australia_ret_1d", "NEE_NextEra_ret_20d", "Industrial_Production_zscore_60d", "SLB_Schlumberger_ret_1d", "heston_var_ev_h3", "HangSeng_HK_ret_5d", "Brent_Oil_FRED_ret_20d", "US5Y_Rate_ret_5d", "Retail_Sales_zscore_60d", "TM_Telephone_vol_20d", "ITT_ITTInc_ret_5d"], "is_new": true}, {"model_id": "new_h3_GLOBAL_RandomForest_N25_t3", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 3, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "HUM_Humana_ret_5d", "INTC_ret_1d", "ITT_ITTInc_ret_5d", "EWY_Korea_ret_20d", "EMR_Emerson_ret_20d", "VVIX_ret_20d", "MS_MorganStanley_ret_1d", "SJM_JM_Smucker_ret_1d", "heston_var_ev_h5", "EWA_Australia_ret_1d", "SBUX_zscore_60d", "vix_acceleration_1d", "US6M_Rate_ret_20d", "IBEX_Spain_ret_20d", "EFFR_ret_1d", "TM_Telephone_vol_20d", "EWH_HongKong_ret_5d", "CLX_Clorox_vol_20d", "SJM_JM_Smucker_ret_5d", "TXN_vol_20d", "TED_Spread_vol_20d", "MS_MorganStanley_ret_5d", "CPB_CampbellSoup_zscore_60d"], "is_new": true}, {"model_id": "new_h3_GLOBAL_RandomForest_N25_t4", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 3, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWS_Singapore_ret_5d", "HangSeng_HK_ret_1d", "ES_Evergy_ret_1d", "SO_SouthernCo_ret_5d", "ASX_Australia_ret_5d", "XLB_Materials_zscore_60d", "CCI_CrownCastle_vol_20d", "CPB_CampbellSoup_ret_20d", "Industrial_Production_zscore_60d", "EXC_Exelon_zscore_60d", "CTAS_Cintas_vol_20d", "MSTR_Bitcoin3_ret_20d", "SLB_Schlumberger_ret_5d", "EOG_EOGResources_vol_20d", "HUM_Humana_ret_5d", "XLF_Fin_vol_20d", "PG_ret_20d", "PCAR_PaccarInc_ret_5d", "3M_vol_20d", "LOW_Lowes_ret_20d", "heston_ev_h3", "US3Y_Rate_ret_5d", "Core_CPI_zscore_60d"], "is_new": true}, {"model_id": "new_h3_GLOBAL_RandomForest_N25_t5", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 3, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "MRK_Merck_zscore_60d", "heston_var_ev_h5", "GE_ret_1d", "SCHW_Schwab_ret_5d", "MS_MorganStanley_zscore_60d", "LMT_LockheedMartin_vol_20d", "CPB_CampbellSoup_ret_20d", "EQR_Equity_ret_1d", "NWL_Newell_ret_20d", "SBUX_ret_5d", "GILD_Gilead_ret_20d", "VVIX_ret_20d", "AMGN_Amgen_ret_1d", "PFE_ret_1d", "LOW_Lowes_ret_20d", "EWM_Malaysia_ret_1d", "LMT_LockheedMartin_ret_1d", "CPB_CampbellSoup_ret_5d", "EWG_Germany_ret_20d", "SO_SouthernCo_ret_5d", "EWY_Korea_ret_20d", "EXC_Exelon_zscore_60d", "LUV_SouthwestAir_ret_5d"], "is_new": true}, {"model_id": "new_h3_GLOBAL_RandomForest_N25_t6", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 3, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "HangSeng_HK_ret_5d", "vix_acceleration_1d", "WTI_Oil_FRED_zscore_60d", "IWM_SmallCap_vol_20d", "NWL_Newell_ret_20d", "IYM_BasicMaterials_ret_20d", "CPB_CampbellSoup_zscore_60d", "IYR_US_REIT2_zscore_60d", "TGT_Target_zscore_60d", "EFFR_ret_1d", "heston_ev_h3", "EWJ_Japan_vol_20d", "BA_ret_1d", "VVIX_ret_20d", "HangSeng_HK_ret_1d", "SBUX_vol_20d", "M_Macys_vol_20d", "ASX_Australia_vol_20d", "JNJ_ret_1d", "US3M_Rate_zscore_60d", "Nikkei_Japan_vol_20d", "PAYX_Paychex_vol_20d", "NEE_NextEra_ret_20d"], "is_new": true}, {"model_id": "new_h3_GLOBAL_RandomForest_N25_t7", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 3, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "ENB_EnbridgeInc_ret_1d", "GE_ret_1d", "US3M_Rate_vol_20d", "T_ret_1d", "EWL_Switzerland_vol_20d", "EWM_Malaysia_vol_20d", "HangSeng_HK_ret_1d", "EXC_Exelon_zscore_60d", "CPB_CampbellSoup_ret_20d", "IBEX_Spain_ret_20d", "MSTR_Bitcoin3_ret_20d", "SBUX_vol_20d", "US6M_Rate_ret_20d", "MSTR_Bitcoin3_ret_5d", "XOM_ret_1d", "EWG_Germany_vol_20d", "SLB_Schlumberger_ret_5d", "VOD_Vodafone_zscore_60d", "DOW_Price_zscore_60d", "MRK_Merck_zscore_60d", "vix_mean_abs_ret_5d", "CPB_CampbellSoup_ret_5d", "HD_ret_5d"], "is_new": true}, {"model_id": "new_h3_GLOBAL_RandomForest_N30_t0", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 3, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "Industrial_Production_zscore_60d", "heston_var_ev_h3", "hmm_p_stress", "spx_momentum_3d", "EFFR_vol_20d", "XLK_Tech_zscore_60d", "AXP_Amex_vol_20d", "SPY_zscore_60d", "EWM_Malaysia_ret_1d", "CI_Cigna_vol_20d", "CCI_CrownCastle_vol_20d", "MS_MorganStanley_ret_5d", "SO_SouthernCo_ret_5d", "NVDA_vol_20d", "PAYX_Paychex_vol_20d", "FedFunds_zscore_60d", "spx_vol_5d", "XLV_Health_zscore_60d", "TED_Spread_vol_20d", "ORCL_vol_20d", "WTI_Oil_FRED_zscore_60d", "PAYX_Paychex_zscore_60d", "EWG_Germany_ret_20d", "DE_Deere_ret_5d", "3M_ret_5d", "PFE_ret_1d", "BTI_BritishAmerican_ret_20d", "TGT_Target_zscore_60d"], "is_new": true}, {"model_id": "new_h3_GLOBAL_RandomForest_N30_t1", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 3, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWY_Korea_zscore_60d", "Core_CPI_zscore_60d", "spx_vol_5d", "T_ret_1d", "DHR_vol_20d", "MS_MorganStanley_ret_5d", "XLB_Materials_zscore_60d", "SJM_JM_Smucker_ret_1d", "CI_Cigna_vol_20d", "SLB_Schlumberger_ret_1d", "PLD_Prologis_ret_5d", "BDX_Becton_Dickinson_ret_20d", "vix_acceleration_1d", "JNJ_ret_1d", "EWM_Malaysia_vol_20d", "US3Y_Rate_ret_5d", "US1Y_Rate_ret_5d", "PG_ret_20d", "GD_GeneralDynamics_zscore_60d", "AMT_AmericanTower_ret_1d", "LMT_LockheedMartin_ret_1d", "GE_ret_1d", "Brent_Oil_FRED_ret_5d", "US5Y_Rate_ret_5d", "BTI_BritishAmerican_ret_20d", "HangSeng_HK_ret_1d", "TGT_Target_zscore_60d", "INTC_ret_5d"], "is_new": true}, {"model_id": "new_h3_GLOBAL_RandomForest_N30_t2", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 3, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "HD_ret_5d", "EOG_EOGResources_ret_5d", "MS_MorganStanley_ret_1d", "CCI_CrownCastle_vol_20d", "IBEX_Spain_ret_20d", "DE_Deere_ret_5d", "DHR_vol_20d", "PAYX_Paychex_ret_20d", "spx_abs_ret_max_5d", "AORD_AUS_zscore_60d", "HUM_Humana_ret_5d", "LOW_Lowes_ret_20d", "spx_momentum_3d", "ENB_EnbridgeInc_ret_1d", "XLY_Disc_vol_20d", "TED_Spread_vol_20d", "IYM_BasicMaterials_ret_20d", "SJM_JM_Smucker_ret_5d", "US3M_Rate_vol_20d", "EWL_Switzerland_vol_20d", "PFE_ret_1d", "heston_ev_h3", "US30Y_Rate_ret_20d", "Core_CPI_zscore_60d", "AXP_Amex_vol_20d", "EWQ_France_ret_20d", "LLY_zscore_60d", "PAYX_Paychex_zscore_60d"], "is_new": true}, {"model_id": "new_h3_GLOBAL_RandomForest_N30_t3", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 3, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "US1Y_Rate_ret_5d", "US3M_Rate_zscore_60d", "Core_CPI_zscore_60d", "ASX_Australia_vol_20d", "AORD_AUS_zscore_60d", "IBEX_Spain_ret_20d", "heston_var_ev_h7", "US7Y_Rate_ret_20d", "EQIX_Equinix_ret_5d", "EXC_Exelon_ret_1d", "M_Macys_vol_20d", "spx_momentum_3d", "Core_PCE_zscore_60d", "DOW_Price_zscore_60d", "TM_Telephone_vol_20d", "LOW_Lowes_ret_20d", "EXC_Exelon_zscore_60d", "vix_acceleration_1d", "US5Y_Rate_ret_5d", "PFE_ret_1d", "heston_var_ev_h3", "gjr_condvar_h1", "EWM_Malaysia_vol_20d", "CPB_CampbellSoup_ret_5d", "IYR_US_REIT2_zscore_60d", "BA_ret_1d", "SO_SouthernCo_ret_5d", "Brent_Oil_FRED_ret_20d"], "is_new": true}, {"model_id": "new_h3_GLOBAL_RandomForest_N30_t4", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 3, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "BA_ret_1d", "GD_GeneralDynamics_zscore_60d", "DE_Deere_ret_5d", "US6M_Rate_ret_20d", "LOW_Lowes_ret_20d", "PAYX_Paychex_vol_20d", "WTI_Oil_FRED_zscore_60d", "AMGN_Amgen_ret_1d", "EWJ_Japan_vol_20d", "ITT_ITTInc_ret_5d", "HD_ret_1d", "AXP_Amex_ret_20d", "MS_MorganStanley_ret_1d", "GE_ret_1d", "DOW_Price_zscore_60d", "TED_Spread_zscore_60d", "EFFR_vol_20d", "EWL_Switzerland_zscore_60d", "LOW_Lowes_ret_5d", "US3M_Rate_zscore_60d", "SJM_JM_Smucker_ret_5d", "gjr_condvar_h1", "TGT_Target_zscore_60d", "VRP_ma5", "SBUX_vol_20d", "MS_MorganStanley_zscore_60d", "CCI_CrownCastle_vol_20d", "TM_Telephone_ret_1d"], "is_new": true}, {"model_id": "new_h3_GLOBAL_RandomForest_N30_t5", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 3, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "Brent_Oil_FRED_ret_5d", "EOG_EOGResources_vol_20d", "CCI_CrownCastle_vol_20d", "DOW_Price_zscore_60d", "EWQ_France_ret_20d", "HD_ret_20d", "US3M_Rate_zscore_60d", "MSTR_Bitcoin3_ret_20d", "ASX_Australia_ret_5d", "LMT_LockheedMartin_ret_1d", "XLB_Materials_zscore_60d", "EWG_Germany_vol_20d", "SJM_JM_Smucker_ret_1d", "AVB_AvalonBay_zscore_60d", "MS_MorganStanley_ret_5d", "DIS_vol_20d", "PG_ret_20d", "DHR_vol_20d", "EWM_Malaysia_ret_1d", "IYR_US_REIT2_zscore_60d", "DHR_ret_1d", "XLV_Health_zscore_60d", "GE_ret_1d", "EWH_HongKong_ret_5d", "heston_var_ev_h5", "CLX_Clorox_vol_20d", "CTAS_Cintas_vol_20d", "EWS_Singapore_ret_5d"], "is_new": true}, {"model_id": "new_h3_GLOBAL_RandomForest_N30_t6", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 3, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "PPL_PPL_ret_1d", "CPB_CampbellSoup_vol_20d", "Nikkei_Japan_zscore_60d", "EMR_Emerson_ret_20d", "XLY_Disc_vol_20d", "SLB_Schlumberger_ret_1d", "HD_ret_1d", "SJM_JM_Smucker_ret_1d", "MRK_Merck_zscore_60d", "HangSeng_HK_ret_5d", "SJM_JM_Smucker_ret_5d", "SCHW_Schwab_ret_5d", "AXP_Amex_ret_20d", "BA_ret_1d", "US3M_Rate_zscore_60d", "spx_vol_5d", "EXC_Exelon_ret_1d", "NFCI_ret_5d", "Core_PCE_zscore_60d", "GD_GeneralDynamics_zscore_60d", "MSTR_Bitcoin3_ret_1d", "US3Y_Rate_ret_5d", "heston_var_ev_h7", "SO_SouthernCo_ret_5d", "EWS_Singapore_ret_5d", "US6M_Rate_ret_20d", "LMT_LockheedMartin_vol_20d", "Michigan_Sentiment_ret_20d"], "is_new": true}, {"model_id": "new_h3_GLOBAL_RandomForest_N30_t7", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 3, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "NWL_Newell_ret_20d", "SJM_JM_Smucker_ret_5d", "Industrial_Production_zscore_60d", "US1Y_Rate_ret_20d", "NFCI_ret_5d", "PCAR_PaccarInc_ret_5d", "ORCL_zscore_60d", "TED_Spread_vol_20d", "XLB_Materials_zscore_60d", "IYR_US_REIT2_zscore_60d", "EWM_Malaysia_zscore_60d", "DHR_ret_1d", "HangSeng_HK_vol_20d", "XLY_Disc_vol_20d", "PAYX_Paychex_zscore_60d", "spx_vol_5d", "EWL_Switzerland_vol_20d", "EWH_HongKong_ret_5d", "AVB_AvalonBay_zscore_60d", "EWM_Malaysia_vol_20d", "MS_MorganStanley_ret_5d", "Brent_Oil_FRED_ret_5d", "US3Y_Rate_ret_5d", "US3M_Rate_vol_20d", "heston_var_ev_h3", "DAX_Germany_zscore_60d", "MSTR_Bitcoin3_ret_1d", "AMT_AmericanTower_ret_1d"], "is_new": true}, {"model_id": "new_h3_GLOBAL_LogisticRegression_N5_t0", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 3, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "ORCL_zscore_60d", "heston_var_ev_h5", "HD_ret_5d"], "is_new": true}, {"model_id": "new_h3_GLOBAL_LogisticRegression_N5_t1", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 3, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "PAYX_Paychex_zscore_60d", "TED_Spread_vol_20d", "AMD_ret_1d"], "is_new": true}, {"model_id": "new_h3_GLOBAL_LogisticRegression_N5_t2", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 3, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "PLD_Prologis_ret_5d", "Brent_Oil_FRED_ret_20d", "T_ret_1d"], "is_new": true}, {"model_id": "new_h3_GLOBAL_LogisticRegression_N5_t3", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 3, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "HD_ret_20d", "Michigan_Sentiment_ret_20d", "Core_CPI_zscore_60d"], "is_new": true}, {"model_id": "new_h3_GLOBAL_LogisticRegression_N5_t4", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 3, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "NFCI_ret_5d", "Industrial_Production_zscore_60d", "HangSeng_HK_ret_5d"], "is_new": true}, {"model_id": "new_h3_GLOBAL_LogisticRegression_N5_t5", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 3, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "MS_MorganStanley_ret_1d", "EWL_Switzerland_zscore_60d", "MSTR_Bitcoin3_ret_5d"], "is_new": true}, {"model_id": "new_h3_GLOBAL_LogisticRegression_N5_t6", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 3, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "US1Y_Rate_ret_20d", "vix_acceleration_1d", "Core_CPI_zscore_60d"], "is_new": true}, {"model_id": "new_h3_GLOBAL_LogisticRegression_N5_t7", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 3, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWH_HongKong_ret_5d", "BDX_Becton_Dickinson_ret_20d", "HD_ret_20d"], "is_new": true}, {"model_id": "new_h3_GLOBAL_LogisticRegression_N8_t0", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 3, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "SBUX_zscore_60d", "VVIX_ret_20d", "US3Y_Rate_ret_5d", "T_ret_1d", "ES_Evergy_ret_1d", "AORD_AUS_zscore_60d"], "is_new": true}, {"model_id": "new_h3_GLOBAL_LogisticRegression_N8_t1", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 3, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "AXP_Amex_ret_20d", "CMCSA_ret_1d", "EOG_EOGResources_vol_20d", "EWM_Malaysia_zscore_60d", "EWG_Germany_ret_20d", "IYR_US_REIT2_zscore_60d"], "is_new": true}, {"model_id": "new_h3_GLOBAL_LogisticRegression_N8_t2", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 3, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "LLY_zscore_60d", "VVIX_ret_20d", "3M_vol_20d", "CCI_CrownCastle_vol_20d", "EQIX_Equinix_ret_5d", "WTI_Oil_FRED_zscore_60d"], "is_new": true}, {"model_id": "new_h3_GLOBAL_LogisticRegression_N8_t3", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 3, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "BTI_BritishAmerican_ret_5d", "EWM_Malaysia_zscore_60d", "EWH_HongKong_ret_5d", "PAYX_Paychex_ret_20d", "AMD_ret_5d", "HD_ret_5d"], "is_new": true}, {"model_id": "new_h3_GLOBAL_LogisticRegression_N8_t4", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 3, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "spx_vol_5d", "US1Y_Rate_ret_20d", "ITT_ITTInc_ret_5d", "AXP_Amex_vol_20d", "PPL_PPL_ret_1d", "HUM_Humana_ret_5d"], "is_new": true}, {"model_id": "new_h3_GLOBAL_LogisticRegression_N8_t5", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 3, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "XLB_Materials_zscore_60d", "EFFR_ret_1d", "TM_Telephone_ret_1d", "DE_Deere_vol_20d", "EWC_Canada_zscore_60d", "NFCI_ret_5d"], "is_new": true}, {"model_id": "new_h3_GLOBAL_LogisticRegression_N8_t6", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 3, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "TXN_vol_20d", "DIS_vol_20d", "Nikkei_Japan_zscore_60d", "US30Y_Rate_ret_20d", "EWM_Malaysia_ret_1d", "CLX_Clorox_vol_20d"], "is_new": true}, {"model_id": "new_h3_GLOBAL_LogisticRegression_N8_t7", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 3, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWY_Korea_ret_20d", "US30Y_Rate_ret_20d", "EWC_Canada_zscore_60d", "EWQ_France_ret_20d", "ORCL_vol_20d", "PCAR_PaccarInc_ret_5d"], "is_new": true}, {"model_id": "new_h3_GLOBAL_LogisticRegression_N10_t0", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 3, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "HangSeng_HK_ret_1d", "BLK_BlackRock_zscore_60d", "HD_ret_20d", "SJM_JM_Smucker_ret_1d", "NFCI_ret_5d", "Brent_Oil_FRED_ret_5d", "XLF_Fin_vol_20d", "HD_ret_5d"], "is_new": true}, {"model_id": "new_h3_GLOBAL_LogisticRegression_N10_t1", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 3, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "TM_Telephone_ret_1d", "CPB_CampbellSoup_zscore_60d", "SLB_Schlumberger_ret_1d", "CTAS_Cintas_vol_20d", "heston_var_ev_h7", "Brent_Oil_FRED_ret_5d", "AMD_ret_1d", "MS_MorganStanley_zscore_60d"], "is_new": true}, {"model_id": "new_h3_GLOBAL_LogisticRegression_N10_t2", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 3, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "ENB_EnbridgeInc_ret_1d", "EWQ_France_ret_20d", "BTI_BritishAmerican_ret_20d", "XLY_Disc_vol_20d", "SBUX_zscore_60d", "SLB_Schlumberger_ret_1d", "US6M_Rate_ret_20d", "JNJ_ret_1d"], "is_new": true}, {"model_id": "new_h3_GLOBAL_LogisticRegression_N10_t3", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 3, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "BTI_BritishAmerican_ret_20d", "ASX_Australia_vol_20d", "US30Y_Rate_ret_20d", "WTI_Oil_FRED_zscore_60d", "SPY_zscore_60d", "Brent_Oil_FRED_ret_5d", "3M_vol_20d", "SBUX_ret_5d"], "is_new": true}, {"model_id": "new_h3_GLOBAL_LogisticRegression_N10_t4", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 3, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "spx_abs_ret_max_5d", "SJM_JM_Smucker_ret_1d", "HangSeng_HK_ret_1d", "ORCL_zscore_60d", "SPY_zscore_60d", "EWQ_France_ret_20d", "ASX_Australia_ret_5d", "CMCSA_ret_1d"], "is_new": true}, {"model_id": "new_h3_GLOBAL_LogisticRegression_N10_t5", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 3, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "3M_vol_20d", "SLB_Schlumberger_ret_5d", "TM_Telephone_ret_1d", "EWG_Germany_vol_20d", "HangSeng_HK_vol_20d", "GD_GeneralDynamics_zscore_60d", "EWY_Korea_zscore_60d", "DOW_Price_zscore_60d"], "is_new": true}, {"model_id": "new_h3_GLOBAL_LogisticRegression_N10_t6", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 3, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "DHR_ret_1d", "GD_GeneralDynamics_zscore_60d", "WTI_Oil_FRED_zscore_60d", "HangSeng_HK_ret_1d", "CMCSA_ret_1d", "SCHW_Schwab_ret_5d", "SJM_JM_Smucker_ret_5d", "IYM_BasicMaterials_ret_20d"], "is_new": true}, {"model_id": "new_h3_GLOBAL_LogisticRegression_N10_t7", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 3, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "AXP_Amex_ret_20d", "EFFR_ret_1d", "EWL_Switzerland_vol_20d", "US7Y_Rate_ret_20d", "DE_Deere_ret_5d", "SLB_Schlumberger_ret_1d", "vix_mean_abs_ret_5d", "heston_var_ev_h3"], "is_new": true}, {"model_id": "new_h3_GLOBAL_LogisticRegression_N12_t0", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 3, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "MRK_Merck_zscore_60d", "ORCL_zscore_60d", "BTI_BritishAmerican_ret_5d", "VVIX_ret_20d", "CMCSA_ret_1d", "AMD_ret_1d", "HD_zscore_60d", "heston_var_ev_h5", "EWQ_France_zscore_60d", "VRP_ma5"], "is_new": true}, {"model_id": "new_h3_GLOBAL_LogisticRegression_N12_t1", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 3, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "CMCSA_ret_1d", "AORD_AUS_zscore_60d", "vix_mean_abs_ret_5d", "EFFR_ret_1d", "HangSeng_HK_ret_5d", "DHR_vol_20d", "LMT_LockheedMartin_ret_1d", "AXP_Amex_ret_20d", "ITT_ITTInc_ret_5d", "XLF_Fin_vol_20d"], "is_new": true}, {"model_id": "new_h3_GLOBAL_LogisticRegression_N12_t2", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 3, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "BDX_Becton_Dickinson_ret_20d", "MSTR_Bitcoin3_ret_1d", "EFFR_vol_20d", "NWL_Newell_ret_20d", "US30Y_Rate_ret_20d", "HUM_Humana_ret_5d", "HD_ret_20d", "BTI_BritishAmerican_ret_20d", "T_ret_1d", "PLD_Prologis_ret_5d"], "is_new": true}, {"model_id": "new_h3_GLOBAL_LogisticRegression_N12_t3", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 3, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "PPL_PPL_ret_1d", "Nikkei_Japan_vol_20d", "heston_ev_h3", "TGT_Target_zscore_60d", "ORCL_zscore_60d", "ES_Evergy_ret_1d", "US3M_Rate_vol_20d", "NOC_Northrop_ret_20d", "DAX_Germany_zscore_60d", "AMT_AmericanTower_ret_1d"], "is_new": true}, {"model_id": "new_h3_GLOBAL_LogisticRegression_N12_t4", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 3, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "TXN_vol_20d", "XOM_ret_20d", "CLX_Clorox_vol_20d", "BDX_Becton_Dickinson_ret_20d", "CMCSA_ret_1d", "IBEX_Spain_ret_20d", "BTI_BritishAmerican_ret_20d", "AXP_Amex_vol_20d", "AORD_AUS_zscore_60d", "TED_Spread_vol_20d"], "is_new": true}, {"model_id": "new_h3_GLOBAL_LogisticRegression_N12_t5", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 3, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWJ_Japan_vol_20d", "AXP_Amex_vol_20d", "MS_MorganStanley_ret_1d", "LLY_zscore_60d", "CPB_CampbellSoup_ret_20d", "Core_CPI_zscore_60d", "Retail_Sales_zscore_60d", "NFCI_ret_5d", "US1Y_Rate_ret_5d", "EMR_Emerson_ret_20d"], "is_new": true}, {"model_id": "new_h3_GLOBAL_LogisticRegression_N12_t6", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 3, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "SLB_Schlumberger_ret_1d", "SO_SouthernCo_ret_5d", "AMZN_ret_5d", "AVB_AvalonBay_zscore_60d", "IBEX_Spain_ret_20d", "XLF_Fin_vol_20d", "NOC_Northrop_ret_20d", "LUV_SouthwestAir_ret_5d", "GILD_Gilead_ret_20d", "NVDA_vol_20d"], "is_new": true}, {"model_id": "new_h3_GLOBAL_LogisticRegression_N12_t7", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 3, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "SLB_Schlumberger_ret_1d", "HangSeng_HK_ret_1d", "heston_var_ev_h5", "VOD_Vodafone_zscore_60d", "IYR_US_REIT2_zscore_60d", "spx_momentum_3d", "CLX_Clorox_vol_20d", "QQQ_vol_20d", "AMD_ret_5d", "EXC_Exelon_zscore_60d"], "is_new": true}, {"model_id": "new_h3_GLOBAL_LogisticRegression_N15_t0", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 3, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWH_HongKong_ret_5d", "AMT_AmericanTower_ret_1d", "TED_Spread_vol_20d", "ORCL_zscore_60d", "US30Y_Rate_ret_20d", "TXN_vol_20d", "IYM_BasicMaterials_ret_20d", "DAX_Germany_zscore_60d", "MSTR_Bitcoin3_ret_5d", "EWM_Malaysia_ret_1d", "EMR_Emerson_ret_20d", "ENB_EnbridgeInc_ret_1d", "T10Y2Y_Spread_ret_5d"], "is_new": true}, {"model_id": "new_h3_GLOBAL_LogisticRegression_N15_t1", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 3, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWM_Malaysia_zscore_60d", "EWA_Australia_zscore_60d", "ITT_ITTInc_ret_5d", "US1Y_Rate_ret_20d", "hmm_p_stress", "BA_ret_1d", "EOG_EOGResources_vol_20d", "Brent_Oil_FRED_ret_20d", "BLK_BlackRock_zscore_60d", "EWL_Switzerland_vol_20d", "Retail_Sales_zscore_60d", "AMD_ret_5d", "AORD_AUS_zscore_60d"], "is_new": true}, {"model_id": "new_h3_GLOBAL_LogisticRegression_N15_t2", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 3, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "TM_Telephone_vol_20d", "DHR_ret_1d", "EWC_Canada_zscore_60d", "IYM_BasicMaterials_ret_20d", "AMZN_ret_5d", "ORCL_vol_20d", "EOG_EOGResources_vol_20d", "PLD_Prologis_ret_5d", "IBEX_Spain_ret_20d", "vix_mean_abs_ret_5d", "CI_Cigna_vol_20d", "MO_AltriaMG_ret_1d", "QQQ_vol_20d"], "is_new": true}, {"model_id": "new_h3_GLOBAL_LogisticRegression_N15_t3", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 3, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "ASX_Australia_vol_20d", "FedFunds_zscore_60d", "CPB_CampbellSoup_ret_20d", "MSTR_Bitcoin3_ret_20d", "CPB_CampbellSoup_ret_5d", "IBEX_Spain_ret_20d", "EWM_Malaysia_ret_1d", "JNJ_ret_1d", "HD_ret_1d", "EWL_Switzerland_zscore_60d", "HangSeng_HK_ret_5d", "US7Y_Rate_ret_20d", "CTAS_Cintas_vol_20d"], "is_new": true}, {"model_id": "new_h3_GLOBAL_LogisticRegression_N15_t4", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 3, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "vix_acceleration_1d", "INTC_ret_1d", "PAYX_Paychex_zscore_60d", "HD_ret_5d", "hmm_p_stress", "NVDA_vol_20d", "XOM_ret_20d", "HUM_Humana_ret_5d", "EWM_Malaysia_zscore_60d", "EWC_Canada_zscore_60d", "BDX_Becton_Dickinson_ret_20d", "GE_ret_1d", "AMGN_Amgen_ret_1d"], "is_new": true}, {"model_id": "new_h3_GLOBAL_LogisticRegression_N15_t5", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 3, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "SJM_JM_Smucker_ret_5d", "Core_PCE_zscore_60d", "LOW_Lowes_ret_5d", "spx_vol_5d", "SO_SouthernCo_ret_5d", "LMT_LockheedMartin_ret_1d", "EWQ_France_ret_20d", "JNJ_ret_1d", "3M_ret_5d", "Nikkei_Japan_zscore_60d", "DAX_Germany_zscore_60d", "XLB_Materials_zscore_60d", "DOW_Price_zscore_60d"], "is_new": true}, {"model_id": "new_h3_GLOBAL_LogisticRegression_N15_t6", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 3, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "US3M_Rate_zscore_60d", "spx_abs_ret_max_5d", "EWY_Korea_zscore_60d", "EWQ_France_zscore_60d", "PAYX_Paychex_ret_20d", "EWC_Canada_zscore_60d", "EWQ_France_ret_20d", "EWJ_Japan_vol_20d", "BTI_BritishAmerican_ret_5d", "EQIX_Equinix_ret_5d", "XLV_Health_zscore_60d", "EWG_Germany_vol_20d", "TED_Spread_zscore_60d"], "is_new": true}, {"model_id": "new_h3_GLOBAL_LogisticRegression_N15_t7", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 3, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "US1Y_Rate_ret_20d", "JNJ_ret_1d", "spx_momentum_3d", "BLK_BlackRock_zscore_60d", "EFFR_vol_20d", "DHR_ret_1d", "AMD_ret_1d", "heston_var_ev_h3", "TGT_Target_zscore_60d", "AMGN_Amgen_ret_1d", "MRK_Merck_zscore_60d", "EWA_Australia_ret_1d", "TXN_vol_20d"], "is_new": true}, {"model_id": "new_h3_GLOBAL_LogisticRegression_N20_t0", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 3, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "PAYX_Paychex_zscore_60d", "VVIX_ret_20d", "MO_AltriaMG_ret_1d", "BTI_BritishAmerican_ret_20d", "EWJ_Japan_vol_20d", "ES_Evergy_ret_1d", "HD_ret_1d", "MS_MorganStanley_zscore_60d", "IWM_SmallCap_vol_20d", "Brent_Oil_FRED_ret_5d", "TXN_vol_20d", "EWA_Australia_ret_1d", "CPB_CampbellSoup_ret_5d", "EXC_Exelon_ret_1d", "ORCL_vol_20d", "Core_CPI_zscore_60d", "gjr_condvar_h1", "ENB_EnbridgeInc_ret_1d"], "is_new": true}, {"model_id": "new_h3_GLOBAL_LogisticRegression_N20_t1", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 3, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "PCAR_PaccarInc_ret_5d", "VOD_Vodafone_zscore_60d", "SLB_Schlumberger_ret_1d", "EWM_Malaysia_ret_1d", "GILD_Gilead_ret_20d", "AMGN_Amgen_ret_1d", "3M_vol_20d", "PAYX_Paychex_zscore_60d", "PAYX_Paychex_ret_20d", "Brent_Oil_FRED_ret_5d", "hmm_p_stress", "NVDA_vol_20d", "gjr_condvar_h1", "BLK_BlackRock_zscore_60d", "CPB_CampbellSoup_ret_5d", "AORD_AUS_zscore_60d", "US30Y_Rate_ret_20d", "EWL_Switzerland_zscore_60d"], "is_new": true}, {"model_id": "new_h3_GLOBAL_LogisticRegression_N20_t2", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 3, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "XLF_Fin_vol_20d", "US3M_Rate_vol_20d", "Nikkei_Japan_zscore_60d", "EWG_Germany_ret_20d", "EOG_EOGResources_vol_20d", "CPB_CampbellSoup_ret_20d", "MO_AltriaMG_ret_1d", "LOW_Lowes_ret_5d", "EWL_Switzerland_zscore_60d", "hmm_p_stress", "EWQ_France_zscore_60d", "XLV_Health_zscore_60d", "PG_ret_20d", "ITT_ITTInc_ret_5d", "CPB_CampbellSoup_vol_20d", "SCHW_Schwab_ret_5d", "IWM_SmallCap_vol_20d", "DE_Deere_ret_5d"], "is_new": true}, {"model_id": "new_h3_GLOBAL_LogisticRegression_N20_t3", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 3, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "spx_vol_5d", "EFFR_vol_20d", "EXC_Exelon_ret_1d", "AMT_AmericanTower_ret_1d", "heston_var_ev_h5", "vix_mean_abs_ret_5d", "AXP_Amex_vol_20d", "ASX_Australia_ret_5d", "DE_Deere_vol_20d", "INTC_ret_1d", "US3Y_Rate_ret_5d", "EWC_Canada_zscore_60d", "SCHW_Schwab_ret_5d", "IYR_US_REIT2_zscore_60d", "TM_Telephone_vol_20d", "CI_Cigna_vol_20d", "TGT_Target_zscore_60d", "EWM_Malaysia_vol_20d"], "is_new": true}, {"model_id": "new_h3_GLOBAL_LogisticRegression_N20_t4", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 3, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "DIS_vol_20d", "US30Y_Rate_ret_20d", "QQQ_vol_20d", "GD_GeneralDynamics_zscore_60d", "AMD_ret_5d", "vix_mean_abs_ret_5d", "TED_Spread_zscore_60d", "VRP_ma5", "SBUX_zscore_60d", "MRK_Merck_zscore_60d", "heston_var_ev_h7", "SBUX_vol_20d", "SPY_zscore_60d", "EXC_Exelon_zscore_60d", "DE_Deere_vol_20d", "EWM_Malaysia_vol_20d", "LLY_zscore_60d", "CCI_CrownCastle_vol_20d"], "is_new": true}, {"model_id": "new_h3_GLOBAL_LogisticRegression_N20_t5", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 3, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "IBEX_Spain_ret_20d", "EQR_Equity_ret_1d", "CTAS_Cintas_vol_20d", "heston_ev_h3", "HangSeng_HK_ret_5d", "US7Y_Rate_ret_20d", "DHR_ret_1d", "TGT_Target_zscore_60d", "SLB_Schlumberger_ret_5d", "HangSeng_HK_ret_1d", "Brent_Oil_FRED_ret_20d", "MS_MorganStanley_ret_1d", "EWM_Malaysia_zscore_60d", "HD_ret_1d", "HD_ret_5d", "SBUX_zscore_60d", "PLD_Prologis_ret_5d", "hmm_p_stress"], "is_new": true}, {"model_id": "new_h3_GLOBAL_LogisticRegression_N20_t6", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 3, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "XLK_Tech_zscore_60d", "EWC_Canada_zscore_60d", "XLY_Disc_vol_20d", "vix_mean_abs_ret_5d", "DOW_Price_zscore_60d", "3M_vol_20d", "LLY_zscore_60d", "MO_AltriaMG_ret_1d", "CCI_CrownCastle_vol_20d", "LUV_SouthwestAir_ret_5d", "hmm_p_stress", "PLD_Prologis_ret_5d", "EWY_Korea_zscore_60d", "LMT_LockheedMartin_vol_20d", "EWA_Australia_ret_1d", "EWM_Malaysia_zscore_60d", "AMD_ret_1d", "HD_zscore_60d"], "is_new": true}, {"model_id": "new_h3_GLOBAL_LogisticRegression_N20_t7", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 3, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "LOW_Lowes_ret_5d", "US3M_Rate_zscore_60d", "US30Y_Rate_ret_20d", "SLB_Schlumberger_ret_5d", "DHR_vol_20d", "MS_MorganStanley_ret_1d", "SPY_zscore_60d", "PPL_PPL_ret_1d", "HD_zscore_60d", "EWL_Switzerland_zscore_60d", "EWQ_France_zscore_60d", "SCHW_Schwab_ret_5d", "EWG_Germany_vol_20d", "CTAS_Cintas_vol_20d", "FedFunds_zscore_60d", "gjr_condvar_h1", "AMD_ret_1d", "XLF_Fin_vol_20d"], "is_new": true}, {"model_id": "new_h3_GLOBAL_LogisticRegression_N25_t0", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 3, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "TXN_vol_20d", "CPB_CampbellSoup_zscore_60d", "3M_vol_20d", "EWL_Switzerland_zscore_60d", "XLF_Fin_vol_20d", "CMCSA_ret_1d", "BTI_BritishAmerican_ret_5d", "Core_CPI_zscore_60d", "HangSeng_HK_ret_1d", "AMD_ret_5d", "SBUX_zscore_60d", "LLY_zscore_60d", "MRK_Merck_zscore_60d", "US3Y_Rate_ret_5d", "PLD_Prologis_ret_5d", "TM_Telephone_ret_1d", "ORCL_vol_20d", "EOG_EOGResources_ret_5d", "Brent_Oil_FRED_ret_20d", "NOC_Northrop_ret_20d", "EWL_Switzerland_vol_20d", "EWY_Korea_zscore_60d", "SLB_Schlumberger_ret_1d"], "is_new": true}, {"model_id": "new_h3_GLOBAL_LogisticRegression_N25_t1", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 3, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "IYR_US_REIT2_zscore_60d", "EWJ_Japan_vol_20d", "HangSeng_HK_ret_5d", "EXC_Exelon_zscore_60d", "heston_ev_h3", "Retail_Sales_zscore_60d", "Industrial_Production_zscore_60d", "LMT_LockheedMartin_vol_20d", "AMT_AmericanTower_ret_1d", "SO_SouthernCo_ret_5d", "DIS_vol_20d", "EWM_Malaysia_vol_20d", "US1Y_Rate_ret_20d", "gjr_condvar_h1", "TED_Spread_zscore_60d", "TGT_Target_zscore_60d", "US1Y_Rate_ret_5d", "CPB_CampbellSoup_ret_20d", "MS_MorganStanley_ret_5d", "CCI_CrownCastle_vol_20d", "EWM_Malaysia_zscore_60d", "PFE_ret_1d", "CI_Cigna_vol_20d"], "is_new": true}, {"model_id": "new_h3_GLOBAL_LogisticRegression_N25_t2", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 3, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWM_Malaysia_ret_1d", "DHR_vol_20d", "SBUX_vol_20d", "DAX_Germany_vol_20d", "HangSeng_HK_vol_20d", "MSTR_Bitcoin3_ret_20d", "vix_mean_abs_ret_5d", "LMT_LockheedMartin_ret_1d", "JNJ_ret_1d", "HUM_Humana_ret_5d", "vix_acceleration_1d", "TGT_Target_zscore_60d", "spx_abs_ret_max_5d", "ORCL_vol_20d", "T10Y2Y_Spread_ret_5d", "NOC_Northrop_ret_20d", "EWJ_Japan_vol_20d", "TXN_vol_20d", "PAYX_Paychex_ret_20d", "EQIX_Equinix_ret_5d", "BA_ret_1d", "EWH_HongKong_ret_5d", "ENB_EnbridgeInc_ret_1d"], "is_new": true}, {"model_id": "new_h3_GLOBAL_LogisticRegression_N25_t3", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 3, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "SBUX_ret_5d", "BLK_BlackRock_zscore_60d", "IBEX_Spain_ret_20d", "XLK_Tech_zscore_60d", "EXC_Exelon_zscore_60d", "EWQ_France_zscore_60d", "DHR_ret_1d", "Retail_Sales_zscore_60d", "HD_zscore_60d", "AORD_AUS_zscore_60d", "EFFR_vol_20d", "MSTR_Bitcoin3_ret_20d", "LMT_LockheedMartin_vol_20d", "JNJ_ret_1d", "T_ret_1d", "ITT_ITTInc_ret_5d", "EQR_Equity_ret_1d", "FedFunds_zscore_60d", "LUV_SouthwestAir_ret_5d", "US30Y_Rate_ret_20d", "AXP_Amex_vol_20d", "ENB_EnbridgeInc_ret_1d", "EWQ_France_ret_20d"], "is_new": true}, {"model_id": "new_h3_GLOBAL_LogisticRegression_N25_t4", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 3, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "SLB_Schlumberger_ret_1d", "CPB_CampbellSoup_ret_5d", "vix_mean_abs_ret_5d", "LMT_LockheedMartin_vol_20d", "heston_var_ev_h7", "HD_ret_1d", "TM_Telephone_ret_1d", "AMZN_ret_5d", "IWM_SmallCap_vol_20d", "CMCSA_ret_1d", "EWY_Korea_zscore_60d", "EWQ_France_ret_20d", "SPY_zscore_60d", "T10Y2Y_Spread_ret_5d", "EQIX_Equinix_ret_5d", "QQQ_vol_20d", "EWM_Malaysia_ret_1d", "SBUX_vol_20d", "GD_GeneralDynamics_zscore_60d", "US6M_Rate_ret_20d", "AXP_Amex_ret_20d", "VOD_Vodafone_zscore_60d", "US3Y_Rate_ret_5d"], "is_new": true}, {"model_id": "new_h3_GLOBAL_LogisticRegression_N25_t5", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 3, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "TGT_Target_zscore_60d", "heston_var_ev_h5", "BTI_BritishAmerican_ret_20d", "EWG_Germany_ret_20d", "DHR_vol_20d", "HUM_Humana_ret_5d", "ASX_Australia_vol_20d", "NEE_NextEra_ret_20d", "DOW_Price_zscore_60d", "Brent_Oil_FRED_ret_20d", "HD_ret_20d", "vix_acceleration_1d", "CPB_CampbellSoup_ret_20d", "EWM_Malaysia_zscore_60d", "EWL_Switzerland_vol_20d", "LMT_LockheedMartin_ret_1d", "EXC_Exelon_zscore_60d", "ORCL_zscore_60d", "EWA_Australia_ret_1d", "EWY_Korea_ret_20d", "PPL_PPL_ret_1d", "BLK_BlackRock_zscore_60d", "EOG_EOGResources_ret_5d"], "is_new": true}, {"model_id": "new_h3_GLOBAL_LogisticRegression_N25_t6", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 3, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWA_Australia_ret_1d", "HD_zscore_60d", "gjr_condvar_h1", "T_ret_1d", "Core_PCE_zscore_60d", "FedFunds_zscore_60d", "Core_CPI_zscore_60d", "LMT_LockheedMartin_vol_20d", "PG_ret_20d", "PCAR_PaccarInc_ret_5d", "CTAS_Cintas_vol_20d", "US1Y_Rate_ret_5d", "BDX_Becton_Dickinson_ret_20d", "PAYX_Paychex_zscore_60d", "EWY_Korea_zscore_60d", "US3M_Rate_vol_20d", "DE_Deere_vol_20d", "DHR_vol_20d", "vix_mean_abs_ret_5d", "Nikkei_Japan_vol_20d", "NOC_Northrop_ret_20d", "LOW_Lowes_ret_5d", "XLF_Fin_vol_20d"], "is_new": true}, {"model_id": "new_h3_GLOBAL_LogisticRegression_N25_t7", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 3, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "CPB_CampbellSoup_zscore_60d", "VRP_ma5", "3M_ret_5d", "SO_SouthernCo_ret_5d", "EQR_Equity_ret_1d", "PAYX_Paychex_vol_20d", "HD_ret_1d", "INTC_ret_5d", "PAYX_Paychex_zscore_60d", "GILD_Gilead_ret_20d", "MRK_Merck_zscore_60d", "NFCI_ret_5d", "LMT_LockheedMartin_ret_1d", "US6M_Rate_ret_20d", "XLB_Materials_zscore_60d", "DOW_Price_zscore_60d", "gjr_condvar_h1", "AMGN_Amgen_ret_1d", "SBUX_vol_20d", "Nikkei_Japan_vol_20d", "EWA_Australia_ret_1d", "EWM_Malaysia_vol_20d", "QQQ_vol_20d"], "is_new": true}, {"model_id": "new_h3_GLOBAL_LogisticRegression_N30_t0", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 3, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWG_Germany_ret_20d", "MRK_Merck_zscore_60d", "EWM_Malaysia_zscore_60d", "SCHW_Schwab_ret_5d", "CTAS_Cintas_vol_20d", "EQR_Equity_ret_1d", "CPB_CampbellSoup_ret_5d", "HD_zscore_60d", "MS_MorganStanley_ret_5d", "EWS_Singapore_ret_5d", "BA_ret_1d", "SJM_JM_Smucker_ret_1d", "BLK_BlackRock_zscore_60d", "BDX_Becton_Dickinson_ret_20d", "XLF_Fin_vol_20d", "IYM_BasicMaterials_ret_20d", "TED_Spread_zscore_60d", "EWG_Germany_vol_20d", "AMT_AmericanTower_ret_1d", "EXC_Exelon_ret_1d", "EOG_EOGResources_ret_5d", "TM_Telephone_vol_20d", "heston_var_ev_h3", "EWL_Switzerland_vol_20d", "DOW_Price_zscore_60d", "EWQ_France_zscore_60d", "BTI_BritishAmerican_ret_20d", "AXP_Amex_vol_20d"], "is_new": true}, {"model_id": "new_h3_GLOBAL_LogisticRegression_N30_t1", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 3, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "SJM_JM_Smucker_ret_1d", "HD_ret_20d", "Retail_Sales_zscore_60d", "FedFunds_zscore_60d", "HD_zscore_60d", "XLB_Materials_zscore_60d", "EWG_Germany_ret_20d", "MS_MorganStanley_zscore_60d", "MS_MorganStanley_ret_5d", "SBUX_zscore_60d", "EWM_Malaysia_vol_20d", "IWM_SmallCap_vol_20d", "DHR_vol_20d", "CPB_CampbellSoup_ret_5d", "M_Macys_vol_20d", "AMGN_Amgen_ret_1d", "T10Y2Y_Spread_ret_5d", "EWY_Korea_ret_20d", "HangSeng_HK_ret_1d", "AXP_Amex_ret_20d", "ENB_EnbridgeInc_ret_1d", "SLB_Schlumberger_ret_1d", "PCAR_PaccarInc_ret_5d", "HD_ret_1d", "3M_ret_5d", "EWJ_Japan_vol_20d", "EWG_Germany_vol_20d", "GD_GeneralDynamics_zscore_60d"], "is_new": true}, {"model_id": "new_h3_GLOBAL_LogisticRegression_N30_t2", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 3, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "WTI_Oil_FRED_zscore_60d", "EWC_Canada_zscore_60d", "LUV_SouthwestAir_ret_5d", "EWL_Switzerland_zscore_60d", "XLY_Disc_vol_20d", "Industrial_Production_zscore_60d", "MS_MorganStanley_ret_1d", "EMR_Emerson_ret_20d", "CPB_CampbellSoup_vol_20d", "IWM_SmallCap_vol_20d", "HD_zscore_60d", "DOW_Price_zscore_60d", "NOC_Northrop_ret_20d", "DE_Deere_vol_20d", "NFCI_ret_5d", "PCAR_PaccarInc_ret_5d", "CCI_CrownCastle_vol_20d", "TED_Spread_vol_20d", "TM_Telephone_ret_1d", "EFFR_vol_20d", "NEE_NextEra_ret_20d", "Retail_Sales_zscore_60d", "BLK_BlackRock_zscore_60d", "CLX_Clorox_vol_20d", "EWH_HongKong_ret_5d", "Nikkei_Japan_zscore_60d", "BTI_BritishAmerican_ret_20d", "CMCSA_ret_1d"], "is_new": true}, {"model_id": "new_h3_GLOBAL_LogisticRegression_N30_t3", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 3, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "ES_Evergy_ret_1d", "DHR_vol_20d", "Core_PCE_zscore_60d", "heston_var_ev_h5", "PFE_ret_1d", "hmm_p_stress", "PAYX_Paychex_vol_20d", "LMT_LockheedMartin_ret_1d", "VOD_Vodafone_zscore_60d", "GILD_Gilead_ret_20d", "BTI_BritishAmerican_ret_5d", "HangSeng_HK_ret_1d", "FedFunds_zscore_60d", "SBUX_vol_20d", "Retail_Sales_zscore_60d", "EQIX_Equinix_ret_5d", "AMZN_ret_5d", "HD_ret_1d", "JNJ_ret_1d", "EWM_Malaysia_ret_1d", "VVIX_ret_20d", "PAYX_Paychex_ret_20d", "MS_MorganStanley_ret_1d", "PCAR_PaccarInc_ret_5d", "CPB_CampbellSoup_ret_5d", "ORCL_vol_20d", "DHR_ret_1d", "spx_abs_ret_max_5d"], "is_new": true}, {"model_id": "new_h3_GLOBAL_LogisticRegression_N30_t4", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 3, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "US5Y_Rate_ret_5d", "NEE_NextEra_ret_20d", "spx_momentum_3d", "AXP_Amex_vol_20d", "M_Macys_vol_20d", "EWL_Switzerland_zscore_60d", "ASX_Australia_vol_20d", "heston_var_ev_h7", "PLD_Prologis_ret_5d", "VOD_Vodafone_zscore_60d", "TED_Spread_zscore_60d", "EOG_EOGResources_vol_20d", "EWH_HongKong_ret_5d", "EWL_Switzerland_vol_20d", "US1Y_Rate_ret_5d", "US6M_Rate_ret_20d", "Brent_Oil_FRED_ret_5d", "QQQ_vol_20d", "SJM_JM_Smucker_ret_5d", "SBUX_zscore_60d", "EWG_Germany_vol_20d", "LUV_SouthwestAir_ret_5d", "NVDA_vol_20d", "hmm_p_stress", "ES_Evergy_ret_1d", "XLF_Fin_vol_20d", "CPB_CampbellSoup_zscore_60d", "WTI_Oil_FRED_zscore_60d"], "is_new": true}, {"model_id": "new_h3_GLOBAL_LogisticRegression_N30_t5", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 3, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "MSTR_Bitcoin3_ret_5d", "heston_var_ev_h5", "US3Y_Rate_ret_5d", "TED_Spread_vol_20d", "Retail_Sales_zscore_60d", "ORCL_zscore_60d", "EWL_Switzerland_zscore_60d", "IYM_BasicMaterials_ret_20d", "Core_CPI_zscore_60d", "AXP_Amex_vol_20d", "LMT_LockheedMartin_vol_20d", "XOM_ret_1d", "PG_ret_20d", "EWJ_Japan_vol_20d", "HangSeng_HK_ret_1d", "XLV_Health_zscore_60d", "NFCI_ret_5d", "T10Y2Y_Spread_ret_5d", "JNJ_ret_1d", "3M_ret_5d", "ASX_Australia_vol_20d", "HUM_Humana_ret_5d", "PAYX_Paychex_vol_20d", "EXC_Exelon_zscore_60d", "PCAR_PaccarInc_ret_5d", "EMR_Emerson_ret_20d", "EWM_Malaysia_zscore_60d", "M_Macys_vol_20d"], "is_new": true}, {"model_id": "new_h3_GLOBAL_LogisticRegression_N30_t6", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 3, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EFFR_ret_1d", "XLK_Tech_zscore_60d", "AVB_AvalonBay_zscore_60d", "TXN_vol_20d", "PAYX_Paychex_ret_20d", "US3Y_Rate_ret_5d", "DE_Deere_ret_5d", "US3M_Rate_zscore_60d", "MS_MorganStanley_ret_5d", "TM_Telephone_vol_20d", "MSTR_Bitcoin3_ret_5d", "EWM_Malaysia_zscore_60d", "Nikkei_Japan_vol_20d", "SBUX_zscore_60d", "PAYX_Paychex_zscore_60d", "XOM_ret_1d", "XOM_ret_20d", "MSTR_Bitcoin3_ret_1d", "gjr_condvar_h1", "EWM_Malaysia_vol_20d", "MRK_Merck_zscore_60d", "DAX_Germany_zscore_60d", "QQQ_vol_20d", "EWQ_France_zscore_60d", "Michigan_Sentiment_ret_20d", "AMD_ret_1d", "AMGN_Amgen_ret_1d", "XLV_Health_zscore_60d"], "is_new": true}, {"model_id": "new_h3_GLOBAL_LogisticRegression_N30_t7", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 3, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "AMD_ret_1d", "hmm_p_stress", "US30Y_Rate_ret_20d", "GILD_Gilead_ret_20d", "Core_CPI_zscore_60d", "EWC_Canada_zscore_60d", "SLB_Schlumberger_ret_5d", "EWM_Malaysia_zscore_60d", "SJM_JM_Smucker_ret_5d", "US3Y_Rate_ret_5d", "BA_ret_1d", "CPB_CampbellSoup_ret_5d", "BTI_BritishAmerican_ret_20d", "PFE_ret_1d", "INTC_ret_5d", "DOW_Price_zscore_60d", "TXN_vol_20d", "DE_Deere_ret_5d", "3M_vol_20d", "PPL_PPL_ret_1d", "AMD_ret_5d", "SJM_JM_Smucker_ret_1d", "SBUX_ret_5d", "US6M_Rate_ret_20d", "heston_var_ev_h3", "TM_Telephone_ret_1d", "M_Macys_vol_20d", "DAX_Germany_vol_20d"], "is_new": true}, {"model_id": "new_h5_CALM_XGBoost_N5_t0", "algo": "XGBoost", "regime": "CALM", "horizon": 5, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "BTI_BritishAmerican_ret_5d", "NOC_Northrop_ret_20d", "Nikkei_Japan_vol_20d"], "is_new": true}, {"model_id": "new_h5_CALM_XGBoost_N5_t1", "algo": "XGBoost", "regime": "CALM", "horizon": 5, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "DE_Deere_ret_5d", "US30Y_Rate_ret_20d", "AMZN_ret_5d"], "is_new": true}, {"model_id": "new_h5_CALM_XGBoost_N5_t2", "algo": "XGBoost", "regime": "CALM", "horizon": 5, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "HUM_Humana_ret_5d", "MS_MorganStanley_ret_1d", "Nikkei_Japan_zscore_60d"], "is_new": true}, {"model_id": "new_h5_CALM_XGBoost_N5_t3", "algo": "XGBoost", "regime": "CALM", "horizon": 5, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "QQQ_vol_20d", "LOW_Lowes_ret_5d", "US30Y_Rate_ret_20d"], "is_new": true}, {"model_id": "new_h5_CALM_XGBoost_N5_t4", "algo": "XGBoost", "regime": "CALM", "horizon": 5, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "Michigan_Sentiment_ret_20d", "LMT_LockheedMartin_vol_20d", "SJM_JM_Smucker_ret_1d"], "is_new": true}, {"model_id": "new_h5_CALM_XGBoost_N5_t5", "algo": "XGBoost", "regime": "CALM", "horizon": 5, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWC_Canada_zscore_60d", "US3Y_Rate_ret_5d", "EOG_EOGResources_vol_20d"], "is_new": true}, {"model_id": "new_h5_CALM_XGBoost_N5_t6", "algo": "XGBoost", "regime": "CALM", "horizon": 5, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "ES_Evergy_ret_1d", "MS_MorganStanley_zscore_60d", "WTI_Oil_FRED_zscore_60d"], "is_new": true}, {"model_id": "new_h5_CALM_XGBoost_N5_t7", "algo": "XGBoost", "regime": "CALM", "horizon": 5, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "US5Y_Rate_ret_5d", "US1Y_Rate_ret_5d", "LUV_SouthwestAir_ret_5d"], "is_new": true}, {"model_id": "new_h5_CALM_XGBoost_N8_t0", "algo": "XGBoost", "regime": "CALM", "horizon": 5, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "PAYX_Paychex_ret_20d", "EWH_HongKong_ret_5d", "INTC_ret_1d", "US3M_Rate_zscore_60d", "US1Y_Rate_ret_20d", "FedFunds_zscore_60d"], "is_new": true}, {"model_id": "new_h5_CALM_XGBoost_N8_t1", "algo": "XGBoost", "regime": "CALM", "horizon": 5, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "HangSeng_HK_ret_1d", "PLD_Prologis_ret_5d", "EFFR_ret_1d", "AXP_Amex_vol_20d", "XLY_Disc_vol_20d", "DE_Deere_vol_20d"], "is_new": true}, {"model_id": "new_h5_CALM_XGBoost_N8_t2", "algo": "XGBoost", "regime": "CALM", "horizon": 5, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "DHR_vol_20d", "VOD_Vodafone_zscore_60d", "NFCI_ret_5d", "PFE_ret_1d", "VVIX_ret_20d", "EWY_Korea_zscore_60d"], "is_new": true}, {"model_id": "new_h5_CALM_XGBoost_N8_t3", "algo": "XGBoost", "regime": "CALM", "horizon": 5, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "NEE_NextEra_ret_20d", "TM_Telephone_vol_20d", "AXP_Amex_ret_20d", "SBUX_zscore_60d", "EWM_Malaysia_zscore_60d", "Core_CPI_zscore_60d"], "is_new": true}, {"model_id": "new_h5_CALM_XGBoost_N8_t4", "algo": "XGBoost", "regime": "CALM", "horizon": 5, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "PAYX_Paychex_vol_20d", "US3M_Rate_zscore_60d", "PPL_PPL_ret_1d", "M_Macys_vol_20d", "MSTR_Bitcoin3_ret_5d", "vix_mean_abs_ret_5d"], "is_new": true}, {"model_id": "new_h5_CALM_XGBoost_N8_t5", "algo": "XGBoost", "regime": "CALM", "horizon": 5, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "AMD_ret_5d", "AMZN_ret_5d", "heston_var_ev_h7", "EWY_Korea_ret_20d", "Nikkei_Japan_zscore_60d", "HangSeng_HK_vol_20d"], "is_new": true}, {"model_id": "new_h5_CALM_XGBoost_N8_t6", "algo": "XGBoost", "regime": "CALM", "horizon": 5, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWY_Korea_ret_20d", "JNJ_ret_1d", "INTC_ret_5d", "EQR_Equity_ret_1d", "DHR_vol_20d", "AORD_AUS_zscore_60d"], "is_new": true}, {"model_id": "new_h5_CALM_XGBoost_N8_t7", "algo": "XGBoost", "regime": "CALM", "horizon": 5, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EXC_Exelon_zscore_60d", "Industrial_Production_zscore_60d", "spx_abs_ret_max_5d", "M_Macys_vol_20d", "ES_Evergy_ret_1d", "heston_ev_h3"], "is_new": true}, {"model_id": "new_h5_CALM_XGBoost_N10_t0", "algo": "XGBoost", "regime": "CALM", "horizon": 5, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EOG_EOGResources_ret_5d", "LOW_Lowes_ret_5d", "EXC_Exelon_zscore_60d", "heston_var_ev_h7", "AMZN_ret_5d", "US5Y_Rate_ret_5d", "EWY_Korea_zscore_60d", "CPB_CampbellSoup_zscore_60d"], "is_new": true}, {"model_id": "new_h5_CALM_XGBoost_N10_t1", "algo": "XGBoost", "regime": "CALM", "horizon": 5, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EXC_Exelon_zscore_60d", "LMT_LockheedMartin_ret_1d", "EFFR_vol_20d", "IYM_BasicMaterials_ret_20d", "INTC_ret_1d", "CPB_CampbellSoup_ret_20d", "PAYX_Paychex_ret_20d", "MS_MorganStanley_ret_1d"], "is_new": true}, {"model_id": "new_h5_CALM_XGBoost_N10_t2", "algo": "XGBoost", "regime": "CALM", "horizon": 5, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "HangSeng_HK_ret_5d", "XLB_Materials_zscore_60d", "ITT_ITTInc_ret_5d", "AMZN_ret_5d", "WTI_Oil_FRED_zscore_60d", "spx_abs_ret_max_5d", "EQIX_Equinix_ret_5d", "NEE_NextEra_ret_20d"], "is_new": true}, {"model_id": "new_h5_CALM_XGBoost_N10_t3", "algo": "XGBoost", "regime": "CALM", "horizon": 5, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EFFR_ret_1d", "BLK_BlackRock_zscore_60d", "spx_vol_5d", "EWA_Australia_zscore_60d", "CPB_CampbellSoup_ret_5d", "HD_ret_5d", "EWC_Canada_zscore_60d", "AMD_ret_1d"], "is_new": true}, {"model_id": "new_h5_CALM_XGBoost_N10_t4", "algo": "XGBoost", "regime": "CALM", "horizon": 5, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EQIX_Equinix_ret_5d", "SLB_Schlumberger_ret_5d", "SJM_JM_Smucker_ret_5d", "LUV_SouthwestAir_ret_5d", "TED_Spread_zscore_60d", "US5Y_Rate_ret_5d", "XOM_ret_1d", "HD_zscore_60d"], "is_new": true}, {"model_id": "new_h5_CALM_XGBoost_N10_t5", "algo": "XGBoost", "regime": "CALM", "horizon": 5, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "WTI_Oil_FRED_zscore_60d", "US5Y_Rate_ret_5d", "T_ret_1d", "Core_CPI_zscore_60d", "US1Y_Rate_ret_20d", "US3M_Rate_vol_20d", "DAX_Germany_zscore_60d", "EWM_Malaysia_ret_1d"], "is_new": true}, {"model_id": "new_h5_CALM_XGBoost_N10_t6", "algo": "XGBoost", "regime": "CALM", "horizon": 5, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "HangSeng_HK_ret_5d", "US6M_Rate_ret_20d", "BLK_BlackRock_zscore_60d", "SCHW_Schwab_ret_5d", "CPB_CampbellSoup_ret_20d", "spx_momentum_3d", "CPB_CampbellSoup_vol_20d", "EWM_Malaysia_ret_1d"], "is_new": true}, {"model_id": "new_h5_CALM_XGBoost_N10_t7", "algo": "XGBoost", "regime": "CALM", "horizon": 5, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "TM_Telephone_vol_20d", "XLV_Health_zscore_60d", "BA_ret_1d", "MSTR_Bitcoin3_ret_5d", "TGT_Target_zscore_60d", "TED_Spread_zscore_60d", "PLD_Prologis_ret_5d", "SO_SouthernCo_ret_5d"], "is_new": true}, {"model_id": "new_h5_CALM_XGBoost_N12_t0", "algo": "XGBoost", "regime": "CALM", "horizon": 5, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "DHR_vol_20d", "heston_var_ev_h5", "PFE_ret_1d", "SBUX_ret_5d", "XLY_Disc_vol_20d", "ASX_Australia_vol_20d", "Industrial_Production_zscore_60d", "EWA_Australia_ret_1d", "vix_mean_abs_ret_5d", "TM_Telephone_vol_20d"], "is_new": true}, {"model_id": "new_h5_CALM_XGBoost_N12_t1", "algo": "XGBoost", "regime": "CALM", "horizon": 5, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "SLB_Schlumberger_ret_1d", "XOM_ret_1d", "JNJ_ret_1d", "EWJ_Japan_vol_20d", "EWY_Korea_zscore_60d", "EQR_Equity_ret_1d", "US5Y_Rate_ret_5d", "PAYX_Paychex_zscore_60d", "heston_var_ev_h5", "BA_ret_1d"], "is_new": true}, {"model_id": "new_h5_CALM_XGBoost_N12_t2", "algo": "XGBoost", "regime": "CALM", "horizon": 5, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWM_Malaysia_zscore_60d", "SO_SouthernCo_ret_5d", "3M_ret_5d", "AORD_AUS_zscore_60d", "EWY_Korea_zscore_60d", "CMCSA_ret_1d", "EXC_Exelon_ret_1d", "vix_acceleration_1d", "EWA_Australia_zscore_60d", "ENB_EnbridgeInc_ret_1d"], "is_new": true}, {"model_id": "new_h5_CALM_XGBoost_N12_t3", "algo": "XGBoost", "regime": "CALM", "horizon": 5, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "DAX_Germany_vol_20d", "AMD_ret_1d", "FedFunds_zscore_60d", "DE_Deere_vol_20d", "EWG_Germany_vol_20d", "DHR_vol_20d", "AMD_ret_5d", "ASX_Australia_ret_5d", "US1Y_Rate_ret_20d", "MS_MorganStanley_ret_1d"], "is_new": true}, {"model_id": "new_h5_CALM_XGBoost_N12_t4", "algo": "XGBoost", "regime": "CALM", "horizon": 5, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWM_Malaysia_ret_1d", "EXC_Exelon_zscore_60d", "US30Y_Rate_ret_20d", "SBUX_vol_20d", "TED_Spread_vol_20d", "BTI_BritishAmerican_ret_5d", "XLV_Health_zscore_60d", "EWM_Malaysia_vol_20d", "HangSeng_HK_ret_1d", "ENB_EnbridgeInc_ret_1d"], "is_new": true}, {"model_id": "new_h5_CALM_XGBoost_N12_t5", "algo": "XGBoost", "regime": "CALM", "horizon": 5, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "heston_var_ev_h7", "HD_ret_20d", "MS_MorganStanley_zscore_60d", "PLD_Prologis_ret_5d", "INTC_ret_1d", "SBUX_vol_20d", "EWL_Switzerland_zscore_60d", "EOG_EOGResources_ret_5d", "INTC_ret_5d", "CI_Cigna_vol_20d"], "is_new": true}, {"model_id": "new_h5_CALM_XGBoost_N12_t6", "algo": "XGBoost", "regime": "CALM", "horizon": 5, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "NVDA_vol_20d", "BTI_BritishAmerican_ret_5d", "PCAR_PaccarInc_ret_5d", "EWA_Australia_ret_1d", "CLX_Clorox_vol_20d", "EQIX_Equinix_ret_5d", "HD_ret_1d", "EWM_Malaysia_ret_1d", "DOW_Price_zscore_60d", "Core_CPI_zscore_60d"], "is_new": true}, {"model_id": "new_h5_CALM_XGBoost_N12_t7", "algo": "XGBoost", "regime": "CALM", "horizon": 5, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "QQQ_vol_20d", "M_Macys_vol_20d", "Core_CPI_zscore_60d", "PG_ret_20d", "GILD_Gilead_ret_20d", "AXP_Amex_vol_20d", "EWM_Malaysia_vol_20d", "IWM_SmallCap_vol_20d", "heston_ev_h3", "EWG_Germany_vol_20d"], "is_new": true}, {"model_id": "new_h5_CALM_XGBoost_N15_t0", "algo": "XGBoost", "regime": "CALM", "horizon": 5, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "MSTR_Bitcoin3_ret_5d", "T10Y2Y_Spread_ret_5d", "GE_ret_1d", "DAX_Germany_vol_20d", "PAYX_Paychex_vol_20d", "EFFR_ret_1d", "SBUX_zscore_60d", "EWH_HongKong_ret_5d", "BDX_Becton_Dickinson_ret_20d", "MS_MorganStanley_ret_5d", "US3M_Rate_vol_20d", "IYM_BasicMaterials_ret_20d", "FedFunds_zscore_60d"], "is_new": true}, {"model_id": "new_h5_CALM_XGBoost_N15_t1", "algo": "XGBoost", "regime": "CALM", "horizon": 5, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWA_Australia_zscore_60d", "3M_vol_20d", "vix_mean_abs_ret_5d", "CPB_CampbellSoup_vol_20d", "INTC_ret_5d", "BDX_Becton_Dickinson_ret_20d", "US1Y_Rate_ret_20d", "heston_var_ev_h5", "US6M_Rate_ret_20d", "DE_Deere_ret_5d", "XLV_Health_zscore_60d", "SLB_Schlumberger_ret_1d", "EWG_Germany_ret_20d"], "is_new": true}, {"model_id": "new_h5_CALM_XGBoost_N15_t2", "algo": "XGBoost", "regime": "CALM", "horizon": 5, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWM_Malaysia_ret_1d", "EWG_Germany_vol_20d", "PG_ret_20d", "AORD_AUS_zscore_60d", "EWY_Korea_zscore_60d", "vix_acceleration_1d", "ITT_ITTInc_ret_5d", "AVB_AvalonBay_zscore_60d", "EWL_Switzerland_vol_20d", "heston_var_ev_h7", "SO_SouthernCo_ret_5d", "NEE_NextEra_ret_20d", "AMGN_Amgen_ret_1d"], "is_new": true}, {"model_id": "new_h5_CALM_XGBoost_N15_t3", "algo": "XGBoost", "regime": "CALM", "horizon": 5, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "ASX_Australia_ret_5d", "NFCI_ret_5d", "DHR_ret_1d", "SJM_JM_Smucker_ret_5d", "PFE_ret_1d", "EWA_Australia_ret_1d", "GILD_Gilead_ret_20d", "IYR_US_REIT2_zscore_60d", "DAX_Germany_vol_20d", "US3Y_Rate_ret_5d", "MO_AltriaMG_ret_1d", "Core_CPI_zscore_60d", "XLV_Health_zscore_60d"], "is_new": true}, {"model_id": "new_h5_CALM_XGBoost_N15_t4", "algo": "XGBoost", "regime": "CALM", "horizon": 5, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "TM_Telephone_vol_20d", "MSTR_Bitcoin3_ret_5d", "SJM_JM_Smucker_ret_1d", "XOM_ret_1d", "EWQ_France_zscore_60d", "Core_PCE_zscore_60d", "MSTR_Bitcoin3_ret_20d", "AVB_AvalonBay_zscore_60d", "heston_var_ev_h7", "EFFR_ret_1d", "QQQ_vol_20d", "SO_SouthernCo_ret_5d", "NWL_Newell_ret_20d"], "is_new": true}, {"model_id": "new_h5_CALM_XGBoost_N15_t5", "algo": "XGBoost", "regime": "CALM", "horizon": 5, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "LOW_Lowes_ret_5d", "MS_MorganStanley_ret_5d", "CPB_CampbellSoup_ret_20d", "DE_Deere_vol_20d", "TED_Spread_zscore_60d", "BDX_Becton_Dickinson_ret_20d", "NEE_NextEra_ret_20d", "XLF_Fin_vol_20d", "LMT_LockheedMartin_ret_1d", "INTC_ret_5d", "EFFR_vol_20d", "DAX_Germany_vol_20d", "CPB_CampbellSoup_vol_20d"], "is_new": true}, {"model_id": "new_h5_CALM_XGBoost_N15_t6", "algo": "XGBoost", "regime": "CALM", "horizon": 5, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "MS_MorganStanley_zscore_60d", "CTAS_Cintas_vol_20d", "IYR_US_REIT2_zscore_60d", "MSTR_Bitcoin3_ret_20d", "MS_MorganStanley_ret_5d", "PPL_PPL_ret_1d", "US1Y_Rate_ret_20d", "Industrial_Production_zscore_60d", "XLV_Health_zscore_60d", "PAYX_Paychex_ret_20d", "CPB_CampbellSoup_vol_20d", "EWJ_Japan_vol_20d", "GD_GeneralDynamics_zscore_60d"], "is_new": true}, {"model_id": "new_h5_CALM_XGBoost_N15_t7", "algo": "XGBoost", "regime": "CALM", "horizon": 5, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "gjr_condvar_h1", "TM_Telephone_ret_1d", "T10Y2Y_Spread_ret_5d", "BTI_BritishAmerican_ret_5d", "AMD_ret_1d", "GE_ret_1d", "EOG_EOGResources_ret_5d", "EWJ_Japan_vol_20d", "US1Y_Rate_ret_20d", "vix_mean_abs_ret_5d", "IWM_SmallCap_vol_20d", "XOM_ret_20d", "EQR_Equity_ret_1d"], "is_new": true}, {"model_id": "new_h5_CALM_XGBoost_N20_t0", "algo": "XGBoost", "regime": "CALM", "horizon": 5, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "LOW_Lowes_ret_20d", "HD_zscore_60d", "LMT_LockheedMartin_vol_20d", "AXP_Amex_ret_20d", "SJM_JM_Smucker_ret_5d", "ITT_ITTInc_ret_5d", "EXC_Exelon_zscore_60d", "TM_Telephone_vol_20d", "SPY_zscore_60d", "IYR_US_REIT2_zscore_60d", "WTI_Oil_FRED_zscore_60d", "SO_SouthernCo_ret_5d", "gjr_condvar_h1", "MS_MorganStanley_ret_1d", "US3Y_Rate_ret_5d", "AXP_Amex_vol_20d", "HangSeng_HK_vol_20d", "EWQ_France_zscore_60d"], "is_new": true}, {"model_id": "new_h5_CALM_XGBoost_N20_t1", "algo": "XGBoost", "regime": "CALM", "horizon": 5, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "HD_ret_1d", "3M_ret_5d", "MS_MorganStanley_zscore_60d", "ASX_Australia_vol_20d", "DIS_vol_20d", "MS_MorganStanley_ret_1d", "US6M_Rate_ret_20d", "ASX_Australia_ret_5d", "US3Y_Rate_ret_5d", "heston_ev_h3", "LUV_SouthwestAir_ret_5d", "VVIX_ret_20d", "HangSeng_HK_vol_20d", "AXP_Amex_vol_20d", "LLY_zscore_60d", "heston_var_ev_h5", "DE_Deere_ret_5d", "CPB_CampbellSoup_vol_20d"], "is_new": true}, {"model_id": "new_h5_CALM_XGBoost_N20_t2", "algo": "XGBoost", "regime": "CALM", "horizon": 5, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "SJM_JM_Smucker_ret_1d", "PLD_Prologis_ret_5d", "EWA_Australia_zscore_60d", "BA_ret_1d", "TGT_Target_zscore_60d", "SJM_JM_Smucker_ret_5d", "CMCSA_ret_1d", "IYM_BasicMaterials_ret_20d", "EWM_Malaysia_vol_20d", "AMT_AmericanTower_ret_1d", "EMR_Emerson_ret_20d", "US6M_Rate_ret_20d", "EWL_Switzerland_zscore_60d", "EWG_Germany_vol_20d", "EWG_Germany_ret_20d", "MS_MorganStanley_ret_1d", "PAYX_Paychex_vol_20d", "BLK_BlackRock_zscore_60d"], "is_new": true}, {"model_id": "new_h5_CALM_XGBoost_N20_t3", "algo": "XGBoost", "regime": "CALM", "horizon": 5, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "LMT_LockheedMartin_vol_20d", "EWM_Malaysia_zscore_60d", "TM_Telephone_vol_20d", "HangSeng_HK_vol_20d", "CPB_CampbellSoup_ret_5d", "DHR_vol_20d", "PAYX_Paychex_vol_20d", "VOD_Vodafone_zscore_60d", "gjr_condvar_h1", "CPB_CampbellSoup_vol_20d", "EWM_Malaysia_ret_1d", "MRK_Merck_zscore_60d", "EWL_Switzerland_zscore_60d", "EWY_Korea_ret_20d", "DHR_ret_1d", "LOW_Lowes_ret_20d", "T10Y2Y_Spread_ret_5d", "US30Y_Rate_ret_20d"], "is_new": true}, {"model_id": "new_h5_CALM_XGBoost_N20_t4", "algo": "XGBoost", "regime": "CALM", "horizon": 5, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "CCI_CrownCastle_vol_20d", "NWL_Newell_ret_20d", "EOG_EOGResources_vol_20d", "SBUX_vol_20d", "GILD_Gilead_ret_20d", "EQIX_Equinix_ret_5d", "DHR_ret_1d", "CPB_CampbellSoup_ret_20d", "DAX_Germany_zscore_60d", "vix_mean_abs_ret_5d", "CTAS_Cintas_vol_20d", "CLX_Clorox_vol_20d", "IWM_SmallCap_vol_20d", "HangSeng_HK_ret_5d", "hmm_p_stress", "GE_ret_1d", "IBEX_Spain_ret_20d", "heston_var_ev_h7"], "is_new": true}, {"model_id": "new_h5_CALM_XGBoost_N20_t5", "algo": "XGBoost", "regime": "CALM", "horizon": 5, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "HangSeng_HK_ret_1d", "NEE_NextEra_ret_20d", "Brent_Oil_FRED_ret_20d", "AMGN_Amgen_ret_1d", "MS_MorganStanley_ret_1d", "TGT_Target_zscore_60d", "CMCSA_ret_1d", "3M_ret_5d", "SCHW_Schwab_ret_5d", "CPB_CampbellSoup_vol_20d", "EWY_Korea_ret_20d", "SBUX_ret_5d", "SBUX_vol_20d", "US6M_Rate_ret_20d", "SBUX_zscore_60d", "EWM_Malaysia_zscore_60d", "AMD_ret_5d", "XLY_Disc_vol_20d"], "is_new": true}, {"model_id": "new_h5_CALM_XGBoost_N20_t6", "algo": "XGBoost", "regime": "CALM", "horizon": 5, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "heston_ev_h3", "MS_MorganStanley_ret_5d", "AMD_ret_1d", "PAYX_Paychex_vol_20d", "EWA_Australia_ret_1d", "SCHW_Schwab_ret_5d", "EOG_EOGResources_vol_20d", "BDX_Becton_Dickinson_ret_20d", "HangSeng_HK_vol_20d", "AVB_AvalonBay_zscore_60d", "EWS_Singapore_ret_5d", "EQIX_Equinix_ret_5d", "HangSeng_HK_ret_1d", "HD_ret_20d", "HangSeng_HK_ret_5d", "AMT_AmericanTower_ret_1d", "CPB_CampbellSoup_vol_20d", "EFFR_vol_20d"], "is_new": true}, {"model_id": "new_h5_CALM_XGBoost_N20_t7", "algo": "XGBoost", "regime": "CALM", "horizon": 5, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "heston_var_ev_h3", "EWG_Germany_vol_20d", "spx_abs_ret_max_5d", "IBEX_Spain_ret_20d", "EWG_Germany_ret_20d", "M_Macys_vol_20d", "BTI_BritishAmerican_ret_5d", "GILD_Gilead_ret_20d", "Industrial_Production_zscore_60d", "LUV_SouthwestAir_ret_5d", "ASX_Australia_ret_5d", "CI_Cigna_vol_20d", "EFFR_vol_20d", "WTI_Oil_FRED_zscore_60d", "AMGN_Amgen_ret_1d", "EWY_Korea_zscore_60d", "EWQ_France_zscore_60d", "TED_Spread_zscore_60d"], "is_new": true}, {"model_id": "new_h5_CALM_XGBoost_N25_t0", "algo": "XGBoost", "regime": "CALM", "horizon": 5, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "ORCL_zscore_60d", "LLY_zscore_60d", "MSTR_Bitcoin3_ret_5d", "HUM_Humana_ret_5d", "M_Macys_vol_20d", "XLK_Tech_zscore_60d", "SJM_JM_Smucker_ret_5d", "EWG_Germany_ret_20d", "MSTR_Bitcoin3_ret_20d", "EWQ_France_zscore_60d", "EWG_Germany_vol_20d", "DOW_Price_zscore_60d", "ASX_Australia_ret_5d", "US3M_Rate_zscore_60d", "HD_ret_20d", "T10Y2Y_Spread_ret_5d", "CPB_CampbellSoup_ret_20d", "NEE_NextEra_ret_20d", "SLB_Schlumberger_ret_1d", "LMT_LockheedMartin_vol_20d", "EWA_Australia_ret_1d", "CI_Cigna_vol_20d", "US7Y_Rate_ret_20d"], "is_new": true}, {"model_id": "new_h5_CALM_XGBoost_N25_t1", "algo": "XGBoost", "regime": "CALM", "horizon": 5, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "BTI_BritishAmerican_ret_5d", "HangSeng_HK_ret_5d", "MSTR_Bitcoin3_ret_20d", "CTAS_Cintas_vol_20d", "SO_SouthernCo_ret_5d", "PPL_PPL_ret_1d", "BDX_Becton_Dickinson_ret_20d", "XOM_ret_20d", "EMR_Emerson_ret_20d", "QQQ_vol_20d", "EWM_Malaysia_vol_20d", "XOM_ret_1d", "DE_Deere_ret_5d", "GD_GeneralDynamics_zscore_60d", "MS_MorganStanley_ret_1d", "PG_ret_20d", "TM_Telephone_ret_1d", "EWM_Malaysia_zscore_60d", "CLX_Clorox_vol_20d", "HangSeng_HK_vol_20d", "XLV_Health_zscore_60d", "HD_ret_1d", "EFFR_vol_20d"], "is_new": true}, {"model_id": "new_h5_CALM_XGBoost_N25_t2", "algo": "XGBoost", "regime": "CALM", "horizon": 5, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "DHR_vol_20d", "FedFunds_zscore_60d", "gjr_condvar_h1", "EWC_Canada_zscore_60d", "XOM_ret_1d", "NFCI_ret_5d", "US3Y_Rate_ret_5d", "Nikkei_Japan_zscore_60d", "XOM_ret_20d", "HangSeng_HK_vol_20d", "PAYX_Paychex_vol_20d", "SBUX_zscore_60d", "EWJ_Japan_vol_20d", "DOW_Price_zscore_60d", "EWY_Korea_ret_20d", "heston_var_ev_h3", "EFFR_vol_20d", "EXC_Exelon_ret_1d", "VRP_ma5", "ENB_EnbridgeInc_ret_1d", "VVIX_ret_20d", "CLX_Clorox_vol_20d", "XLV_Health_zscore_60d"], "is_new": true}, {"model_id": "new_h5_CALM_XGBoost_N25_t3", "algo": "XGBoost", "regime": "CALM", "horizon": 5, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "MS_MorganStanley_ret_5d", "Brent_Oil_FRED_ret_20d", "SPY_zscore_60d", "Michigan_Sentiment_ret_20d", "XLV_Health_zscore_60d", "heston_var_ev_h7", "HD_ret_5d", "spx_momentum_3d", "AMT_AmericanTower_ret_1d", "LOW_Lowes_ret_20d", "MS_MorganStanley_zscore_60d", "VOD_Vodafone_zscore_60d", "DOW_Price_zscore_60d", "PAYX_Paychex_vol_20d", "EWY_Korea_zscore_60d", "MO_AltriaMG_ret_1d", "AMZN_ret_5d", "LUV_SouthwestAir_ret_5d", "NEE_NextEra_ret_20d", "heston_ev_h3", "TED_Spread_vol_20d", "ENB_EnbridgeInc_ret_1d", "MSTR_Bitcoin3_ret_20d"], "is_new": true}, {"model_id": "new_h5_CALM_XGBoost_N25_t4", "algo": "XGBoost", "regime": "CALM", "horizon": 5, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "GILD_Gilead_ret_20d", "EWL_Switzerland_zscore_60d", "XOM_ret_20d", "T10Y2Y_Spread_ret_5d", "Brent_Oil_FRED_ret_20d", "EFFR_ret_1d", "Industrial_Production_zscore_60d", "US30Y_Rate_ret_20d", "AXP_Amex_vol_20d", "ES_Evergy_ret_1d", "INTC_ret_5d", "MS_MorganStanley_ret_5d", "EOG_EOGResources_ret_5d", "CI_Cigna_vol_20d", "WTI_Oil_FRED_zscore_60d", "PAYX_Paychex_ret_20d", "GE_ret_1d", "QQQ_vol_20d", "ORCL_vol_20d", "heston_var_ev_h3", "HangSeng_HK_ret_1d", "VRP_ma5", "BDX_Becton_Dickinson_ret_20d"], "is_new": true}, {"model_id": "new_h5_CALM_XGBoost_N25_t5", "algo": "XGBoost", "regime": "CALM", "horizon": 5, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "HD_ret_5d", "DIS_vol_20d", "NOC_Northrop_ret_20d", "INTC_ret_1d", "MSTR_Bitcoin3_ret_1d", "EFFR_ret_1d", "NWL_Newell_ret_20d", "Brent_Oil_FRED_ret_20d", "MS_MorganStanley_zscore_60d", "BDX_Becton_Dickinson_ret_20d", "EXC_Exelon_ret_1d", "Michigan_Sentiment_ret_20d", "HangSeng_HK_ret_1d", "EWY_Korea_ret_20d", "AMZN_ret_5d", "XLV_Health_zscore_60d", "ORCL_zscore_60d", "EMR_Emerson_ret_20d", "HangSeng_HK_vol_20d", "PG_ret_20d", "MS_MorganStanley_ret_5d", "MS_MorganStanley_ret_1d", "AMD_ret_1d"], "is_new": true}, {"model_id": "new_h5_CALM_XGBoost_N25_t6", "algo": "XGBoost", "regime": "CALM", "horizon": 5, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "XLV_Health_zscore_60d", "SBUX_ret_5d", "TED_Spread_zscore_60d", "EWH_HongKong_ret_5d", "PG_ret_20d", "HD_ret_20d", "Retail_Sales_zscore_60d", "MSTR_Bitcoin3_ret_1d", "MSTR_Bitcoin3_ret_20d", "XLB_Materials_zscore_60d", "EWM_Malaysia_ret_1d", "SCHW_Schwab_ret_5d", "PPL_PPL_ret_1d", "vix_acceleration_1d", "ES_Evergy_ret_1d", "HangSeng_HK_vol_20d", "EQIX_Equinix_ret_5d", "EXC_Exelon_ret_1d", "EOG_EOGResources_ret_5d", "XLY_Disc_vol_20d", "SLB_Schlumberger_ret_5d", "IBEX_Spain_ret_20d", "Brent_Oil_FRED_ret_20d"], "is_new": true}, {"model_id": "new_h5_CALM_XGBoost_N25_t7", "algo": "XGBoost", "regime": "CALM", "horizon": 5, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "DE_Deere_vol_20d", "SLB_Schlumberger_ret_1d", "DIS_vol_20d", "Nikkei_Japan_vol_20d", "CPB_CampbellSoup_zscore_60d", "EFFR_vol_20d", "HD_ret_1d", "US7Y_Rate_ret_20d", "XLK_Tech_zscore_60d", "US1Y_Rate_ret_20d", "PG_ret_20d", "PAYX_Paychex_vol_20d", "XLF_Fin_vol_20d", "US3M_Rate_zscore_60d", "WTI_Oil_FRED_zscore_60d", "HD_ret_5d", "vix_acceleration_1d", "Brent_Oil_FRED_ret_20d", "heston_var_ev_h5", "IBEX_Spain_ret_20d", "ASX_Australia_vol_20d", "NEE_NextEra_ret_20d", "heston_ev_h3"], "is_new": true}, {"model_id": "new_h5_CALM_XGBoost_N30_t0", "algo": "XGBoost", "regime": "CALM", "horizon": 5, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "M_Macys_vol_20d", "CMCSA_ret_1d", "AMD_ret_1d", "HUM_Humana_ret_5d", "ASX_Australia_vol_20d", "US30Y_Rate_ret_20d", "NWL_Newell_ret_20d", "PPL_PPL_ret_1d", "EWA_Australia_ret_1d", "DHR_vol_20d", "ITT_ITTInc_ret_5d", "T10Y2Y_Spread_ret_5d", "MS_MorganStanley_ret_5d", "SJM_JM_Smucker_ret_5d", "GD_GeneralDynamics_zscore_60d", "XOM_ret_20d", "Brent_Oil_FRED_ret_20d", "AORD_AUS_zscore_60d", "MSTR_Bitcoin3_ret_1d", "EFFR_vol_20d", "GILD_Gilead_ret_20d", "PFE_ret_1d", "Brent_Oil_FRED_ret_5d", "MS_MorganStanley_ret_1d", "LLY_zscore_60d", "XOM_ret_1d", "EWA_Australia_zscore_60d", "TM_Telephone_ret_1d"], "is_new": true}, {"model_id": "new_h5_CALM_XGBoost_N30_t1", "algo": "XGBoost", "regime": "CALM", "horizon": 5, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EQR_Equity_ret_1d", "AMZN_ret_5d", "Industrial_Production_zscore_60d", "FedFunds_zscore_60d", "CTAS_Cintas_vol_20d", "LMT_LockheedMartin_vol_20d", "DAX_Germany_zscore_60d", "SJM_JM_Smucker_ret_5d", "vix_acceleration_1d", "SBUX_ret_5d", "heston_var_ev_h5", "NVDA_vol_20d", "HangSeng_HK_ret_1d", "PG_ret_20d", "heston_ev_h3", "3M_vol_20d", "Core_CPI_zscore_60d", "XOM_ret_20d", "US6M_Rate_ret_20d", "MSTR_Bitcoin3_ret_1d", "DHR_ret_1d", "AMGN_Amgen_ret_1d", "DOW_Price_zscore_60d", "EWQ_France_zscore_60d", "EWA_Australia_zscore_60d", "MSTR_Bitcoin3_ret_20d", "heston_var_ev_h7", "SBUX_zscore_60d"], "is_new": true}, {"model_id": "new_h5_CALM_XGBoost_N30_t2", "algo": "XGBoost", "regime": "CALM", "horizon": 5, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "US30Y_Rate_ret_20d", "Brent_Oil_FRED_ret_20d", "EWH_HongKong_ret_5d", "PPL_PPL_ret_1d", "TM_Telephone_vol_20d", "VRP_ma5", "IYM_BasicMaterials_ret_20d", "MS_MorganStanley_ret_1d", "DHR_ret_1d", "EWG_Germany_ret_20d", "IWM_SmallCap_vol_20d", "HUM_Humana_ret_5d", "EWM_Malaysia_ret_1d", "LUV_SouthwestAir_ret_5d", "BTI_BritishAmerican_ret_5d", "EMR_Emerson_ret_20d", "VOD_Vodafone_zscore_60d", "AMGN_Amgen_ret_1d", "IBEX_Spain_ret_20d", "LOW_Lowes_ret_5d", "TXN_vol_20d", "US3M_Rate_vol_20d", "NOC_Northrop_ret_20d", "EWL_Switzerland_zscore_60d", "US1Y_Rate_ret_20d", "LMT_LockheedMartin_vol_20d", "XLY_Disc_vol_20d", "heston_var_ev_h3"], "is_new": true}, {"model_id": "new_h5_CALM_XGBoost_N30_t3", "algo": "XGBoost", "regime": "CALM", "horizon": 5, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "BDX_Becton_Dickinson_ret_20d", "HD_ret_1d", "INTC_ret_1d", "CLX_Clorox_vol_20d", "AMT_AmericanTower_ret_1d", "M_Macys_vol_20d", "heston_var_ev_h5", "MSTR_Bitcoin3_ret_20d", "NEE_NextEra_ret_20d", "AMD_ret_5d", "MRK_Merck_zscore_60d", "US1Y_Rate_ret_5d", "SO_SouthernCo_ret_5d", "AXP_Amex_ret_20d", "US6M_Rate_ret_20d", "spx_vol_5d", "US3M_Rate_zscore_60d", "EFFR_vol_20d", "EXC_Exelon_zscore_60d", "EWJ_Japan_vol_20d", "XLV_Health_zscore_60d", "SJM_JM_Smucker_ret_5d", "CI_Cigna_vol_20d", "Industrial_Production_zscore_60d", "TED_Spread_vol_20d", "SLB_Schlumberger_ret_5d", "EWY_Korea_ret_20d", "EMR_Emerson_ret_20d"], "is_new": true}, {"model_id": "new_h5_CALM_XGBoost_N30_t4", "algo": "XGBoost", "regime": "CALM", "horizon": 5, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EOG_EOGResources_ret_5d", "CTAS_Cintas_vol_20d", "CPB_CampbellSoup_zscore_60d", "NFCI_ret_5d", "TM_Telephone_vol_20d", "MS_MorganStanley_ret_1d", "Michigan_Sentiment_ret_20d", "AXP_Amex_ret_20d", "EMR_Emerson_ret_20d", "TM_Telephone_ret_1d", "XLF_Fin_vol_20d", "EQR_Equity_ret_1d", "TED_Spread_vol_20d", "CMCSA_ret_1d", "EXC_Exelon_ret_1d", "SLB_Schlumberger_ret_1d", "gjr_condvar_h1", "MS_MorganStanley_ret_5d", "LMT_LockheedMartin_ret_1d", "ORCL_vol_20d", "VOD_Vodafone_zscore_60d", "CPB_CampbellSoup_ret_20d", "SBUX_ret_5d", "US1Y_Rate_ret_20d", "vix_acceleration_1d", "LUV_SouthwestAir_ret_5d", "EXC_Exelon_zscore_60d", "US6M_Rate_ret_20d"], "is_new": true}, {"model_id": "new_h5_CALM_XGBoost_N30_t5", "algo": "XGBoost", "regime": "CALM", "horizon": 5, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "BTI_BritishAmerican_ret_5d", "INTC_ret_1d", "EWQ_France_zscore_60d", "US1Y_Rate_ret_5d", "EWL_Switzerland_zscore_60d", "spx_abs_ret_max_5d", "MSTR_Bitcoin3_ret_1d", "MS_MorganStanley_ret_1d", "heston_var_ev_h3", "HangSeng_HK_vol_20d", "HD_ret_20d", "VRP_ma5", "EWC_Canada_zscore_60d", "gjr_condvar_h1", "PLD_Prologis_ret_5d", "XLY_Disc_vol_20d", "EWM_Malaysia_vol_20d", "BA_ret_1d", "EWM_Malaysia_zscore_60d", "HUM_Humana_ret_5d", "DHR_vol_20d", "Nikkei_Japan_zscore_60d", "XLV_Health_zscore_60d", "T_ret_1d", "GD_GeneralDynamics_zscore_60d", "JNJ_ret_1d", "MS_MorganStanley_zscore_60d", "ORCL_vol_20d"], "is_new": true}, {"model_id": "new_h5_CALM_XGBoost_N30_t6", "algo": "XGBoost", "regime": "CALM", "horizon": 5, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "LMT_LockheedMartin_ret_1d", "M_Macys_vol_20d", "US30Y_Rate_ret_20d", "heston_var_ev_h7", "Brent_Oil_FRED_ret_5d", "NEE_NextEra_ret_20d", "3M_ret_5d", "VVIX_ret_20d", "MSTR_Bitcoin3_ret_5d", "HangSeng_HK_vol_20d", "BDX_Becton_Dickinson_ret_20d", "MO_AltriaMG_ret_1d", "ORCL_zscore_60d", "SLB_Schlumberger_ret_5d", "SCHW_Schwab_ret_5d", "Nikkei_Japan_vol_20d", "NOC_Northrop_ret_20d", "AORD_AUS_zscore_60d", "spx_abs_ret_max_5d", "PPL_PPL_ret_1d", "TXN_vol_20d", "EFFR_vol_20d", "MSTR_Bitcoin3_ret_20d", "CPB_CampbellSoup_ret_20d", "HD_ret_20d", "heston_var_ev_h5", "EWM_Malaysia_vol_20d", "AMGN_Amgen_ret_1d"], "is_new": true}, {"model_id": "new_h5_CALM_XGBoost_N30_t7", "algo": "XGBoost", "regime": "CALM", "horizon": 5, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "PCAR_PaccarInc_ret_5d", "WTI_Oil_FRED_zscore_60d", "SJM_JM_Smucker_ret_1d", "DE_Deere_vol_20d", "spx_vol_5d", "EWG_Germany_ret_20d", "AMD_ret_5d", "CTAS_Cintas_vol_20d", "VOD_Vodafone_zscore_60d", "XLV_Health_zscore_60d", "Nikkei_Japan_vol_20d", "XOM_ret_1d", "US7Y_Rate_ret_20d", "HangSeng_HK_vol_20d", "MS_MorganStanley_zscore_60d", "MRK_Merck_zscore_60d", "EFFR_ret_1d", "CPB_CampbellSoup_zscore_60d", "QQQ_vol_20d", "HD_zscore_60d", "XLK_Tech_zscore_60d", "MS_MorganStanley_ret_5d", "INTC_ret_1d", "US1Y_Rate_ret_5d", "TXN_vol_20d", "TM_Telephone_ret_1d", "GD_GeneralDynamics_zscore_60d", "EWC_Canada_zscore_60d"], "is_new": true}, {"model_id": "new_h5_CALM_LightGBM_N5_t0", "algo": "LightGBM", "regime": "CALM", "horizon": 5, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWM_Malaysia_ret_1d", "US1Y_Rate_ret_5d", "AMT_AmericanTower_ret_1d"], "is_new": true}, {"model_id": "new_h5_CALM_LightGBM_N5_t1", "algo": "LightGBM", "regime": "CALM", "horizon": 5, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "HangSeng_HK_ret_5d", "MRK_Merck_zscore_60d", "PLD_Prologis_ret_5d"], "is_new": true}, {"model_id": "new_h5_CALM_LightGBM_N5_t2", "algo": "LightGBM", "regime": "CALM", "horizon": 5, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWC_Canada_zscore_60d", "US1Y_Rate_ret_20d", "BTI_BritishAmerican_ret_20d"], "is_new": true}, {"model_id": "new_h5_CALM_LightGBM_N5_t3", "algo": "LightGBM", "regime": "CALM", "horizon": 5, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "heston_var_ev_h5", "DE_Deere_ret_5d", "TED_Spread_zscore_60d"], "is_new": true}, {"model_id": "new_h5_CALM_LightGBM_N5_t4", "algo": "LightGBM", "regime": "CALM", "horizon": 5, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWA_Australia_ret_1d", "hmm_p_stress", "AMT_AmericanTower_ret_1d"], "is_new": true}, {"model_id": "new_h5_CALM_LightGBM_N5_t5", "algo": "LightGBM", "regime": "CALM", "horizon": 5, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "Michigan_Sentiment_ret_20d", "EWL_Switzerland_vol_20d", "VVIX_ret_20d"], "is_new": true}, {"model_id": "new_h5_CALM_LightGBM_N5_t6", "algo": "LightGBM", "regime": "CALM", "horizon": 5, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWG_Germany_vol_20d", "US5Y_Rate_ret_5d", "PAYX_Paychex_zscore_60d"], "is_new": true}, {"model_id": "new_h5_CALM_LightGBM_N5_t7", "algo": "LightGBM", "regime": "CALM", "horizon": 5, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "US7Y_Rate_ret_20d", "NEE_NextEra_ret_20d", "HUM_Humana_ret_5d"], "is_new": true}, {"model_id": "new_h5_CALM_LightGBM_N8_t0", "algo": "LightGBM", "regime": "CALM", "horizon": 5, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "LOW_Lowes_ret_20d", "EWM_Malaysia_vol_20d", "GD_GeneralDynamics_zscore_60d", "ASX_Australia_vol_20d", "US6M_Rate_ret_20d", "Nikkei_Japan_zscore_60d"], "is_new": true}, {"model_id": "new_h5_CALM_LightGBM_N8_t1", "algo": "LightGBM", "regime": "CALM", "horizon": 5, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "CLX_Clorox_vol_20d", "SBUX_vol_20d", "INTC_ret_1d", "NVDA_vol_20d", "EWL_Switzerland_vol_20d", "HD_ret_20d"], "is_new": true}, {"model_id": "new_h5_CALM_LightGBM_N8_t2", "algo": "LightGBM", "regime": "CALM", "horizon": 5, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EFFR_ret_1d", "Core_PCE_zscore_60d", "ASX_Australia_ret_5d", "VOD_Vodafone_zscore_60d", "CPB_CampbellSoup_ret_5d", "MSTR_Bitcoin3_ret_5d"], "is_new": true}, {"model_id": "new_h5_CALM_LightGBM_N8_t3", "algo": "LightGBM", "regime": "CALM", "horizon": 5, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "Retail_Sales_zscore_60d", "EMR_Emerson_ret_20d", "PCAR_PaccarInc_ret_5d", "SLB_Schlumberger_ret_1d", "EWQ_France_zscore_60d", "EWJ_Japan_vol_20d"], "is_new": true}, {"model_id": "new_h5_CALM_LightGBM_N8_t4", "algo": "LightGBM", "regime": "CALM", "horizon": 5, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "SBUX_zscore_60d", "EWG_Germany_ret_20d", "BTI_BritishAmerican_ret_20d", "US3M_Rate_vol_20d", "NFCI_ret_5d", "SPY_zscore_60d"], "is_new": true}, {"model_id": "new_h5_CALM_LightGBM_N8_t5", "algo": "LightGBM", "regime": "CALM", "horizon": 5, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "US6M_Rate_ret_20d", "SJM_JM_Smucker_ret_5d", "Brent_Oil_FRED_ret_20d", "DAX_Germany_zscore_60d", "AXP_Amex_ret_20d", "Nikkei_Japan_zscore_60d"], "is_new": true}, {"model_id": "new_h5_CALM_LightGBM_N8_t6", "algo": "LightGBM", "regime": "CALM", "horizon": 5, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "PFE_ret_1d", "Core_PCE_zscore_60d", "MO_AltriaMG_ret_1d", "NWL_Newell_ret_20d", "US7Y_Rate_ret_20d", "CLX_Clorox_vol_20d"], "is_new": true}, {"model_id": "new_h5_CALM_LightGBM_N8_t7", "algo": "LightGBM", "regime": "CALM", "horizon": 5, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "Brent_Oil_FRED_ret_20d", "EFFR_vol_20d", "XLY_Disc_vol_20d", "TXN_vol_20d", "EWJ_Japan_vol_20d", "VOD_Vodafone_zscore_60d"], "is_new": true}, {"model_id": "new_h5_CALM_LightGBM_N10_t0", "algo": "LightGBM", "regime": "CALM", "horizon": 5, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "CCI_CrownCastle_vol_20d", "MSTR_Bitcoin3_ret_20d", "GILD_Gilead_ret_20d", "MSTR_Bitcoin3_ret_5d", "AMZN_ret_5d", "SBUX_vol_20d", "CPB_CampbellSoup_zscore_60d", "LMT_LockheedMartin_ret_1d"], "is_new": true}, {"model_id": "new_h5_CALM_LightGBM_N10_t1", "algo": "LightGBM", "regime": "CALM", "horizon": 5, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "LLY_zscore_60d", "ENB_EnbridgeInc_ret_1d", "T_ret_1d", "IYM_BasicMaterials_ret_20d", "EWG_Germany_ret_20d", "AXP_Amex_vol_20d", "ASX_Australia_ret_5d", "EXC_Exelon_zscore_60d"], "is_new": true}, {"model_id": "new_h5_CALM_LightGBM_N10_t2", "algo": "LightGBM", "regime": "CALM", "horizon": 5, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "TGT_Target_zscore_60d", "IYR_US_REIT2_zscore_60d", "XLY_Disc_vol_20d", "EQR_Equity_ret_1d", "DOW_Price_zscore_60d", "DAX_Germany_zscore_60d", "BA_ret_1d", "AMGN_Amgen_ret_1d"], "is_new": true}, {"model_id": "new_h5_CALM_LightGBM_N10_t3", "algo": "LightGBM", "regime": "CALM", "horizon": 5, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "HangSeng_HK_vol_20d", "heston_var_ev_h5", "Core_CPI_zscore_60d", "HangSeng_HK_ret_1d", "GE_ret_1d", "DE_Deere_vol_20d", "heston_var_ev_h3", "BDX_Becton_Dickinson_ret_20d"], "is_new": true}, {"model_id": "new_h5_CALM_LightGBM_N10_t4", "algo": "LightGBM", "regime": "CALM", "horizon": 5, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "DHR_vol_20d", "3M_ret_5d", "MSTR_Bitcoin3_ret_1d", "T_ret_1d", "PLD_Prologis_ret_5d", "EOG_EOGResources_vol_20d", "BTI_BritishAmerican_ret_5d", "SJM_JM_Smucker_ret_1d"], "is_new": true}, {"model_id": "new_h5_CALM_LightGBM_N10_t5", "algo": "LightGBM", "regime": "CALM", "horizon": 5, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "TED_Spread_vol_20d", "PPL_PPL_ret_1d", "heston_var_ev_h5", "AORD_AUS_zscore_60d", "spx_vol_5d", "vix_mean_abs_ret_5d", "US1Y_Rate_ret_5d", "Core_PCE_zscore_60d"], "is_new": true}, {"model_id": "new_h5_CALM_LightGBM_N10_t6", "algo": "LightGBM", "regime": "CALM", "horizon": 5, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "CCI_CrownCastle_vol_20d", "ORCL_vol_20d", "SBUX_ret_5d", "CPB_CampbellSoup_zscore_60d", "DIS_vol_20d", "EWQ_France_ret_20d", "AVB_AvalonBay_zscore_60d", "FedFunds_zscore_60d"], "is_new": true}, {"model_id": "new_h5_CALM_LightGBM_N10_t7", "algo": "LightGBM", "regime": "CALM", "horizon": 5, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "LUV_SouthwestAir_ret_5d", "XLB_Materials_zscore_60d", "XLF_Fin_vol_20d", "TM_Telephone_vol_20d", "3M_ret_5d", "gjr_condvar_h1", "CPB_CampbellSoup_ret_20d", "NEE_NextEra_ret_20d"], "is_new": true}, {"model_id": "new_h5_CALM_LightGBM_N12_t0", "algo": "LightGBM", "regime": "CALM", "horizon": 5, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWM_Malaysia_ret_1d", "NWL_Newell_ret_20d", "NEE_NextEra_ret_20d", "EMR_Emerson_ret_20d", "ENB_EnbridgeInc_ret_1d", "EFFR_ret_1d", "ASX_Australia_vol_20d", "TXN_vol_20d", "MSTR_Bitcoin3_ret_1d", "BTI_BritishAmerican_ret_20d"], "is_new": true}, {"model_id": "new_h5_CALM_LightGBM_N12_t1", "algo": "LightGBM", "regime": "CALM", "horizon": 5, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWQ_France_ret_20d", "AMT_AmericanTower_ret_1d", "LUV_SouthwestAir_ret_5d", "AMZN_ret_5d", "US3Y_Rate_ret_5d", "MSTR_Bitcoin3_ret_20d", "XLB_Materials_zscore_60d", "EXC_Exelon_zscore_60d", "SBUX_vol_20d", "T_ret_1d"], "is_new": true}, {"model_id": "new_h5_CALM_LightGBM_N12_t2", "algo": "LightGBM", "regime": "CALM", "horizon": 5, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "MS_MorganStanley_zscore_60d", "3M_vol_20d", "US1Y_Rate_ret_5d", "DIS_vol_20d", "EXC_Exelon_zscore_60d", "PAYX_Paychex_ret_20d", "HangSeng_HK_ret_5d", "AMD_ret_5d", "PLD_Prologis_ret_5d", "US3M_Rate_zscore_60d"], "is_new": true}, {"model_id": "new_h5_CALM_LightGBM_N12_t3", "algo": "LightGBM", "regime": "CALM", "horizon": 5, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "IBEX_Spain_ret_20d", "HD_ret_20d", "CTAS_Cintas_vol_20d", "vix_mean_abs_ret_5d", "PLD_Prologis_ret_5d", "MO_AltriaMG_ret_1d", "MSTR_Bitcoin3_ret_1d", "Nikkei_Japan_vol_20d", "MSTR_Bitcoin3_ret_5d", "CLX_Clorox_vol_20d"], "is_new": true}, {"model_id": "new_h5_CALM_LightGBM_N12_t4", "algo": "LightGBM", "regime": "CALM", "horizon": 5, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWL_Switzerland_zscore_60d", "EOG_EOGResources_ret_5d", "EWQ_France_ret_20d", "US7Y_Rate_ret_20d", "EQR_Equity_ret_1d", "XLY_Disc_vol_20d", "MRK_Merck_zscore_60d", "AMD_ret_1d", "Core_CPI_zscore_60d", "VOD_Vodafone_zscore_60d"], "is_new": true}, {"model_id": "new_h5_CALM_LightGBM_N12_t5", "algo": "LightGBM", "regime": "CALM", "horizon": 5, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWA_Australia_zscore_60d", "DE_Deere_vol_20d", "CPB_CampbellSoup_vol_20d", "MS_MorganStanley_ret_5d", "Industrial_Production_zscore_60d", "SJM_JM_Smucker_ret_5d", "3M_vol_20d", "US1Y_Rate_ret_5d", "PFE_ret_1d", "DOW_Price_zscore_60d"], "is_new": true}, {"model_id": "new_h5_CALM_LightGBM_N12_t6", "algo": "LightGBM", "regime": "CALM", "horizon": 5, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "spx_momentum_3d", "Brent_Oil_FRED_ret_5d", "Core_CPI_zscore_60d", "XLY_Disc_vol_20d", "EWM_Malaysia_ret_1d", "TED_Spread_vol_20d", "TM_Telephone_ret_1d", "EWJ_Japan_vol_20d", "BLK_BlackRock_zscore_60d", "XLB_Materials_zscore_60d"], "is_new": true}, {"model_id": "new_h5_CALM_LightGBM_N12_t7", "algo": "LightGBM", "regime": "CALM", "horizon": 5, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "MS_MorganStanley_zscore_60d", "vix_acceleration_1d", "PPL_PPL_ret_1d", "ES_Evergy_ret_1d", "WTI_Oil_FRED_zscore_60d", "CI_Cigna_vol_20d", "EWS_Singapore_ret_5d", "AMD_ret_1d", "GE_ret_1d", "DE_Deere_ret_5d"], "is_new": true}, {"model_id": "new_h5_CALM_LightGBM_N15_t0", "algo": "LightGBM", "regime": "CALM", "horizon": 5, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "LMT_LockheedMartin_ret_1d", "EFFR_vol_20d", "Retail_Sales_zscore_60d", "EWM_Malaysia_vol_20d", "CTAS_Cintas_vol_20d", "CCI_CrownCastle_vol_20d", "T_ret_1d", "ORCL_zscore_60d", "MS_MorganStanley_ret_5d", "NFCI_ret_5d", "SBUX_zscore_60d", "ES_Evergy_ret_1d", "GILD_Gilead_ret_20d"], "is_new": true}, {"model_id": "new_h5_CALM_LightGBM_N15_t1", "algo": "LightGBM", "regime": "CALM", "horizon": 5, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "AMZN_ret_5d", "EWY_Korea_ret_20d", "SPY_zscore_60d", "IBEX_Spain_ret_20d", "DAX_Germany_vol_20d", "BA_ret_1d", "CPB_CampbellSoup_ret_20d", "US3M_Rate_zscore_60d", "LMT_LockheedMartin_vol_20d", "US6M_Rate_ret_20d", "XLB_Materials_zscore_60d", "XOM_ret_20d", "ES_Evergy_ret_1d"], "is_new": true}, {"model_id": "new_h5_CALM_LightGBM_N15_t2", "algo": "LightGBM", "regime": "CALM", "horizon": 5, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "GE_ret_1d", "EXC_Exelon_ret_1d", "FedFunds_zscore_60d", "AORD_AUS_zscore_60d", "M_Macys_vol_20d", "Nikkei_Japan_zscore_60d", "CI_Cigna_vol_20d", "LLY_zscore_60d", "US3Y_Rate_ret_5d", "gjr_condvar_h1", "EQIX_Equinix_ret_5d", "DAX_Germany_zscore_60d", "DE_Deere_vol_20d"], "is_new": true}, {"model_id": "new_h5_CALM_LightGBM_N15_t3", "algo": "LightGBM", "regime": "CALM", "horizon": 5, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "TM_Telephone_vol_20d", "MSTR_Bitcoin3_ret_5d", "BTI_BritishAmerican_ret_20d", "US1Y_Rate_ret_20d", "EWA_Australia_ret_1d", "US30Y_Rate_ret_20d", "EOG_EOGResources_ret_5d", "LMT_LockheedMartin_ret_1d", "Industrial_Production_zscore_60d", "VOD_Vodafone_zscore_60d", "Core_PCE_zscore_60d", "HUM_Humana_ret_5d", "EWL_Switzerland_zscore_60d"], "is_new": true}, {"model_id": "new_h5_CALM_LightGBM_N15_t4", "algo": "LightGBM", "regime": "CALM", "horizon": 5, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EOG_EOGResources_ret_5d", "HD_ret_1d", "LMT_LockheedMartin_ret_1d", "MSTR_Bitcoin3_ret_1d", "BLK_BlackRock_zscore_60d", "LLY_zscore_60d", "Brent_Oil_FRED_ret_5d", "SO_SouthernCo_ret_5d", "EOG_EOGResources_vol_20d", "BDX_Becton_Dickinson_ret_20d", "TED_Spread_vol_20d", "heston_var_ev_h3", "ASX_Australia_vol_20d"], "is_new": true}, {"model_id": "new_h5_CALM_LightGBM_N15_t5", "algo": "LightGBM", "regime": "CALM", "horizon": 5, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "SO_SouthernCo_ret_5d", "3M_ret_5d", "TED_Spread_vol_20d", "SCHW_Schwab_ret_5d", "CTAS_Cintas_vol_20d", "EWG_Germany_ret_20d", "JNJ_ret_1d", "EWM_Malaysia_ret_1d", "SPY_zscore_60d", "HangSeng_HK_vol_20d", "PAYX_Paychex_ret_20d", "EWC_Canada_zscore_60d", "DOW_Price_zscore_60d"], "is_new": true}, {"model_id": "new_h5_CALM_LightGBM_N15_t6", "algo": "LightGBM", "regime": "CALM", "horizon": 5, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "PG_ret_20d", "BLK_BlackRock_zscore_60d", "EWM_Malaysia_ret_1d", "Nikkei_Japan_zscore_60d", "LUV_SouthwestAir_ret_5d", "US3M_Rate_zscore_60d", "MRK_Merck_zscore_60d", "US6M_Rate_ret_20d", "ES_Evergy_ret_1d", "INTC_ret_1d", "Retail_Sales_zscore_60d", "AMZN_ret_5d", "SCHW_Schwab_ret_5d"], "is_new": true}, {"model_id": "new_h5_CALM_LightGBM_N15_t7", "algo": "LightGBM", "regime": "CALM", "horizon": 5, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "WTI_Oil_FRED_zscore_60d", "3M_ret_5d", "FedFunds_zscore_60d", "Nikkei_Japan_zscore_60d", "US5Y_Rate_ret_5d", "XLV_Health_zscore_60d", "TM_Telephone_ret_1d", "Brent_Oil_FRED_ret_20d", "BTI_BritishAmerican_ret_5d", "ORCL_zscore_60d", "DHR_vol_20d", "CI_Cigna_vol_20d", "hmm_p_stress"], "is_new": true}, {"model_id": "new_h5_CALM_LightGBM_N20_t0", "algo": "LightGBM", "regime": "CALM", "horizon": 5, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "Nikkei_Japan_zscore_60d", "Brent_Oil_FRED_ret_20d", "MSTR_Bitcoin3_ret_5d", "SBUX_zscore_60d", "CMCSA_ret_1d", "QQQ_vol_20d", "EQR_Equity_ret_1d", "SJM_JM_Smucker_ret_1d", "BA_ret_1d", "DOW_Price_zscore_60d", "DIS_vol_20d", "HD_zscore_60d", "CCI_CrownCastle_vol_20d", "ASX_Australia_vol_20d", "ENB_EnbridgeInc_ret_1d", "NOC_Northrop_ret_20d", "EMR_Emerson_ret_20d", "CPB_CampbellSoup_ret_20d"], "is_new": true}, {"model_id": "new_h5_CALM_LightGBM_N20_t1", "algo": "LightGBM", "regime": "CALM", "horizon": 5, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "ES_Evergy_ret_1d", "AMGN_Amgen_ret_1d", "US6M_Rate_ret_20d", "CLX_Clorox_vol_20d", "HangSeng_HK_ret_5d", "EWM_Malaysia_ret_1d", "heston_var_ev_h3", "DAX_Germany_vol_20d", "US5Y_Rate_ret_5d", "EWQ_France_ret_20d", "TM_Telephone_ret_1d", "EXC_Exelon_ret_1d", "BTI_BritishAmerican_ret_5d", "IYM_BasicMaterials_ret_20d", "EWY_Korea_ret_20d", "CMCSA_ret_1d", "SPY_zscore_60d", "VVIX_ret_20d"], "is_new": true}, {"model_id": "new_h5_CALM_LightGBM_N20_t2", "algo": "LightGBM", "regime": "CALM", "horizon": 5, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWM_Malaysia_zscore_60d", "CCI_CrownCastle_vol_20d", "EWM_Malaysia_vol_20d", "VOD_Vodafone_zscore_60d", "PG_ret_20d", "DHR_vol_20d", "US3M_Rate_zscore_60d", "TM_Telephone_vol_20d", "SPY_zscore_60d", "PPL_PPL_ret_1d", "EQIX_Equinix_ret_5d", "TED_Spread_zscore_60d", "PAYX_Paychex_zscore_60d", "Brent_Oil_FRED_ret_20d", "EWH_HongKong_ret_5d", "AVB_AvalonBay_zscore_60d", "HangSeng_HK_ret_5d", "SLB_Schlumberger_ret_1d"], "is_new": true}, {"model_id": "new_h5_CALM_LightGBM_N20_t3", "algo": "LightGBM", "regime": "CALM", "horizon": 5, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "AMD_ret_5d", "SBUX_ret_5d", "PFE_ret_1d", "XLV_Health_zscore_60d", "Nikkei_Japan_vol_20d", "HD_ret_20d", "EFFR_vol_20d", "PCAR_PaccarInc_ret_5d", "ASX_Australia_ret_5d", "XLF_Fin_vol_20d", "PG_ret_20d", "TED_Spread_zscore_60d", "MO_AltriaMG_ret_1d", "DOW_Price_zscore_60d", "MRK_Merck_zscore_60d", "EWS_Singapore_ret_5d", "VOD_Vodafone_zscore_60d", "XLY_Disc_vol_20d"], "is_new": true}, {"model_id": "new_h5_CALM_LightGBM_N20_t4", "algo": "LightGBM", "regime": "CALM", "horizon": 5, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "BTI_BritishAmerican_ret_5d", "EWM_Malaysia_ret_1d", "DAX_Germany_vol_20d", "PFE_ret_1d", "XOM_ret_20d", "hmm_p_stress", "EXC_Exelon_zscore_60d", "EWY_Korea_zscore_60d", "TM_Telephone_ret_1d", "heston_var_ev_h5", "IYR_US_REIT2_zscore_60d", "US7Y_Rate_ret_20d", "LOW_Lowes_ret_20d", "HD_ret_1d", "EOG_EOGResources_ret_5d", "vix_mean_abs_ret_5d", "EWG_Germany_vol_20d", "EXC_Exelon_ret_1d"], "is_new": true}, {"model_id": "new_h5_CALM_LightGBM_N20_t5", "algo": "LightGBM", "regime": "CALM", "horizon": 5, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "IYM_BasicMaterials_ret_20d", "BTI_BritishAmerican_ret_20d", "AMD_ret_1d", "TED_Spread_vol_20d", "XLF_Fin_vol_20d", "DOW_Price_zscore_60d", "heston_ev_h3", "CPB_CampbellSoup_vol_20d", "PAYX_Paychex_ret_20d", "CPB_CampbellSoup_ret_20d", "3M_ret_5d", "T10Y2Y_Spread_ret_5d", "vix_acceleration_1d", "PAYX_Paychex_zscore_60d", "GILD_Gilead_ret_20d", "TM_Telephone_ret_1d", "IWM_SmallCap_vol_20d", "EWS_Singapore_ret_5d"], "is_new": true}, {"model_id": "new_h5_CALM_LightGBM_N20_t6", "algo": "LightGBM", "regime": "CALM", "horizon": 5, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "MS_MorganStanley_zscore_60d", "AMD_ret_1d", "US30Y_Rate_ret_20d", "PAYX_Paychex_vol_20d", "DE_Deere_vol_20d", "Industrial_Production_zscore_60d", "EWY_Korea_zscore_60d", "SJM_JM_Smucker_ret_1d", "SCHW_Schwab_ret_5d", "IYM_BasicMaterials_ret_20d", "EWQ_France_ret_20d", "SLB_Schlumberger_ret_5d", "T_ret_1d", "Core_PCE_zscore_60d", "PAYX_Paychex_zscore_60d", "ORCL_vol_20d", "TM_Telephone_ret_1d", "EWM_Malaysia_ret_1d"], "is_new": true}, {"model_id": "new_h5_CALM_LightGBM_N20_t7", "algo": "LightGBM", "regime": "CALM", "horizon": 5, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "PPL_PPL_ret_1d", "TED_Spread_vol_20d", "CCI_CrownCastle_vol_20d", "CTAS_Cintas_vol_20d", "AMD_ret_1d", "spx_vol_5d", "Core_PCE_zscore_60d", "Nikkei_Japan_vol_20d", "DOW_Price_zscore_60d", "Core_CPI_zscore_60d", "XLF_Fin_vol_20d", "T10Y2Y_Spread_ret_5d", "US3M_Rate_zscore_60d", "Retail_Sales_zscore_60d", "EXC_Exelon_ret_1d", "HD_ret_20d", "NVDA_vol_20d", "US6M_Rate_ret_20d"], "is_new": true}, {"model_id": "new_h5_CALM_LightGBM_N25_t0", "algo": "LightGBM", "regime": "CALM", "horizon": 5, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "LLY_zscore_60d", "Brent_Oil_FRED_ret_20d", "MS_MorganStanley_ret_1d", "HD_ret_5d", "XLV_Health_zscore_60d", "Core_CPI_zscore_60d", "spx_abs_ret_max_5d", "AMGN_Amgen_ret_1d", "SCHW_Schwab_ret_5d", "AXP_Amex_vol_20d", "gjr_condvar_h1", "GE_ret_1d", "HD_ret_1d", "JNJ_ret_1d", "hmm_p_stress", "WTI_Oil_FRED_zscore_60d", "MSTR_Bitcoin3_ret_1d", "spx_momentum_3d", "US5Y_Rate_ret_5d", "EQR_Equity_ret_1d", "EWQ_France_zscore_60d", "AMZN_ret_5d", "MSTR_Bitcoin3_ret_20d"], "is_new": true}, {"model_id": "new_h5_CALM_LightGBM_N25_t1", "algo": "LightGBM", "regime": "CALM", "horizon": 5, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "Core_CPI_zscore_60d", "Retail_Sales_zscore_60d", "spx_abs_ret_max_5d", "GE_ret_1d", "SPY_zscore_60d", "HUM_Humana_ret_5d", "EMR_Emerson_ret_20d", "INTC_ret_5d", "heston_var_ev_h7", "GD_GeneralDynamics_zscore_60d", "NEE_NextEra_ret_20d", "PG_ret_20d", "Core_PCE_zscore_60d", "WTI_Oil_FRED_zscore_60d", "EWL_Switzerland_vol_20d", "BLK_BlackRock_zscore_60d", "MSTR_Bitcoin3_ret_20d", "hmm_p_stress", "SBUX_vol_20d", "EWC_Canada_zscore_60d", "US6M_Rate_ret_20d", "AMZN_ret_5d", "TED_Spread_zscore_60d"], "is_new": true}, {"model_id": "new_h5_CALM_LightGBM_N25_t2", "algo": "LightGBM", "regime": "CALM", "horizon": 5, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "TGT_Target_zscore_60d", "spx_abs_ret_max_5d", "US3M_Rate_vol_20d", "US30Y_Rate_ret_20d", "EWG_Germany_vol_20d", "EMR_Emerson_ret_20d", "GE_ret_1d", "EWH_HongKong_ret_5d", "IYM_BasicMaterials_ret_20d", "XLB_Materials_zscore_60d", "LOW_Lowes_ret_5d", "Core_CPI_zscore_60d", "EWQ_France_ret_20d", "EWA_Australia_ret_1d", "EWM_Malaysia_ret_1d", "BTI_BritishAmerican_ret_5d", "NEE_NextEra_ret_20d", "VVIX_ret_20d", "MRK_Merck_zscore_60d", "GD_GeneralDynamics_zscore_60d", "EWC_Canada_zscore_60d", "DE_Deere_vol_20d", "EWS_Singapore_ret_5d"], "is_new": true}, {"model_id": "new_h5_CALM_LightGBM_N25_t3", "algo": "LightGBM", "regime": "CALM", "horizon": 5, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "LMT_LockheedMartin_ret_1d", "LOW_Lowes_ret_20d", "DAX_Germany_vol_20d", "NVDA_vol_20d", "XLV_Health_zscore_60d", "SJM_JM_Smucker_ret_1d", "US1Y_Rate_ret_20d", "SPY_zscore_60d", "EWA_Australia_zscore_60d", "hmm_p_stress", "QQQ_vol_20d", "SBUX_ret_5d", "EFFR_vol_20d", "BA_ret_1d", "HangSeng_HK_vol_20d", "EWM_Malaysia_vol_20d", "US3M_Rate_vol_20d", "heston_ev_h3", "Core_PCE_zscore_60d", "SJM_JM_Smucker_ret_5d", "SO_SouthernCo_ret_5d", "T_ret_1d", "GE_ret_1d"], "is_new": true}, {"model_id": "new_h5_CALM_LightGBM_N25_t4", "algo": "LightGBM", "regime": "CALM", "horizon": 5, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "XOM_ret_1d", "NWL_Newell_ret_20d", "INTC_ret_1d", "NOC_Northrop_ret_20d", "HD_ret_20d", "LMT_LockheedMartin_vol_20d", "LOW_Lowes_ret_5d", "EWL_Switzerland_vol_20d", "WTI_Oil_FRED_zscore_60d", "EMR_Emerson_ret_20d", "T10Y2Y_Spread_ret_5d", "EXC_Exelon_zscore_60d", "AMGN_Amgen_ret_1d", "EWG_Germany_vol_20d", "GD_GeneralDynamics_zscore_60d", "SBUX_vol_20d", "Nikkei_Japan_zscore_60d", "heston_var_ev_h3", "M_Macys_vol_20d", "HangSeng_HK_ret_1d", "MO_AltriaMG_ret_1d", "XLV_Health_zscore_60d", "HangSeng_HK_vol_20d"], "is_new": true}, {"model_id": "new_h5_CALM_LightGBM_N25_t5", "algo": "LightGBM", "regime": "CALM", "horizon": 5, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "DOW_Price_zscore_60d", "JNJ_ret_1d", "EWL_Switzerland_vol_20d", "EMR_Emerson_ret_20d", "SO_SouthernCo_ret_5d", "EWA_Australia_ret_1d", "MS_MorganStanley_zscore_60d", "US6M_Rate_ret_20d", "EWM_Malaysia_zscore_60d", "3M_vol_20d", "XOM_ret_20d", "US1Y_Rate_ret_20d", "AORD_AUS_zscore_60d", "Brent_Oil_FRED_ret_20d", "CPB_CampbellSoup_ret_5d", "TM_Telephone_vol_20d", "SBUX_zscore_60d", "EOG_EOGResources_vol_20d", "Core_CPI_zscore_60d", "hmm_p_stress", "US3M_Rate_zscore_60d", "TXN_vol_20d", "VVIX_ret_20d"], "is_new": true}, {"model_id": "new_h5_CALM_LightGBM_N25_t6", "algo": "LightGBM", "regime": "CALM", "horizon": 5, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "AORD_AUS_zscore_60d", "LMT_LockheedMartin_vol_20d", "Nikkei_Japan_vol_20d", "XOM_ret_20d", "heston_var_ev_h7", "JNJ_ret_1d", "PAYX_Paychex_vol_20d", "ENB_EnbridgeInc_ret_1d", "EWG_Germany_ret_20d", "TM_Telephone_ret_1d", "spx_vol_5d", "DAX_Germany_vol_20d", "NEE_NextEra_ret_20d", "AXP_Amex_vol_20d", "Brent_Oil_FRED_ret_5d", "TXN_vol_20d", "gjr_condvar_h1", "BTI_BritishAmerican_ret_5d", "US6M_Rate_ret_20d", "CPB_CampbellSoup_zscore_60d", "EWS_Singapore_ret_5d", "Nikkei_Japan_zscore_60d", "GE_ret_1d"], "is_new": true}, {"model_id": "new_h5_CALM_LightGBM_N25_t7", "algo": "LightGBM", "regime": "CALM", "horizon": 5, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "LOW_Lowes_ret_5d", "Brent_Oil_FRED_ret_5d", "SO_SouthernCo_ret_5d", "IBEX_Spain_ret_20d", "AXP_Amex_ret_20d", "HUM_Humana_ret_5d", "heston_var_ev_h5", "SCHW_Schwab_ret_5d", "SBUX_vol_20d", "EWQ_France_zscore_60d", "LMT_LockheedMartin_vol_20d", "ASX_Australia_vol_20d", "EWQ_France_ret_20d", "Core_PCE_zscore_60d", "spx_vol_5d", "TED_Spread_zscore_60d", "HangSeng_HK_ret_5d", "VOD_Vodafone_zscore_60d", "PFE_ret_1d", "hmm_p_stress", "BTI_BritishAmerican_ret_20d", "MS_MorganStanley_ret_1d", "EWM_Malaysia_ret_1d"], "is_new": true}, {"model_id": "new_h5_CALM_LightGBM_N30_t0", "algo": "LightGBM", "regime": "CALM", "horizon": 5, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "NEE_NextEra_ret_20d", "EWM_Malaysia_ret_1d", "EWM_Malaysia_vol_20d", "NWL_Newell_ret_20d", "vix_mean_abs_ret_5d", "3M_vol_20d", "ES_Evergy_ret_1d", "vix_acceleration_1d", "IYR_US_REIT2_zscore_60d", "DIS_vol_20d", "CMCSA_ret_1d", "SJM_JM_Smucker_ret_1d", "US3Y_Rate_ret_5d", "DE_Deere_vol_20d", "CPB_CampbellSoup_zscore_60d", "Core_PCE_zscore_60d", "US1Y_Rate_ret_5d", "EOG_EOGResources_vol_20d", "TGT_Target_zscore_60d", "XOM_ret_1d", "BTI_BritishAmerican_ret_5d", "EWY_Korea_ret_20d", "AORD_AUS_zscore_60d", "ITT_ITTInc_ret_5d", "Brent_Oil_FRED_ret_20d", "AMT_AmericanTower_ret_1d", "EFFR_ret_1d", "PCAR_PaccarInc_ret_5d"], "is_new": true}, {"model_id": "new_h5_CALM_LightGBM_N30_t1", "algo": "LightGBM", "regime": "CALM", "horizon": 5, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "FedFunds_zscore_60d", "PFE_ret_1d", "XOM_ret_20d", "US5Y_Rate_ret_5d", "NVDA_vol_20d", "BA_ret_1d", "IYR_US_REIT2_zscore_60d", "MSTR_Bitcoin3_ret_1d", "DHR_ret_1d", "EWS_Singapore_ret_5d", "EWL_Switzerland_vol_20d", "ASX_Australia_vol_20d", "VRP_ma5", "EXC_Exelon_ret_1d", "vix_acceleration_1d", "spx_momentum_3d", "PAYX_Paychex_zscore_60d", "Michigan_Sentiment_ret_20d", "SO_SouthernCo_ret_5d", "TED_Spread_vol_20d", "Industrial_Production_zscore_60d", "GD_GeneralDynamics_zscore_60d", "EWC_Canada_zscore_60d", "EXC_Exelon_zscore_60d", "MSTR_Bitcoin3_ret_5d", "GILD_Gilead_ret_20d", "IWM_SmallCap_vol_20d", "SLB_Schlumberger_ret_1d"], "is_new": true}, {"model_id": "new_h5_CALM_LightGBM_N30_t2", "algo": "LightGBM", "regime": "CALM", "horizon": 5, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWG_Germany_vol_20d", "Retail_Sales_zscore_60d", "EWH_HongKong_ret_5d", "EWY_Korea_ret_20d", "EWL_Switzerland_zscore_60d", "AMD_ret_1d", "AVB_AvalonBay_zscore_60d", "EWM_Malaysia_ret_1d", "3M_vol_20d", "GE_ret_1d", "PCAR_PaccarInc_ret_5d", "CPB_CampbellSoup_ret_5d", "MSTR_Bitcoin3_ret_20d", "Michigan_Sentiment_ret_20d", "DOW_Price_zscore_60d", "EWQ_France_ret_20d", "HD_ret_20d", "EQIX_Equinix_ret_5d", "LUV_SouthwestAir_ret_5d", "spx_momentum_3d", "AXP_Amex_ret_20d", "Core_PCE_zscore_60d", "DAX_Germany_zscore_60d", "TED_Spread_vol_20d", "CLX_Clorox_vol_20d", "HD_ret_5d", "AMD_ret_5d", "3M_ret_5d"], "is_new": true}, {"model_id": "new_h5_CALM_LightGBM_N30_t3", "algo": "LightGBM", "regime": "CALM", "horizon": 5, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "DE_Deere_vol_20d", "BA_ret_1d", "SBUX_vol_20d", "SJM_JM_Smucker_ret_5d", "CPB_CampbellSoup_vol_20d", "TM_Telephone_ret_1d", "SCHW_Schwab_ret_5d", "HD_zscore_60d", "CPB_CampbellSoup_ret_5d", "HangSeng_HK_ret_5d", "MSTR_Bitcoin3_ret_5d", "LMT_LockheedMartin_ret_1d", "NOC_Northrop_ret_20d", "XOM_ret_1d", "EWQ_France_ret_20d", "PAYX_Paychex_zscore_60d", "AXP_Amex_ret_20d", "heston_var_ev_h7", "ASX_Australia_vol_20d", "EXC_Exelon_zscore_60d", "XLV_Health_zscore_60d", "QQQ_vol_20d", "EWA_Australia_zscore_60d", "PCAR_PaccarInc_ret_5d", "MO_AltriaMG_ret_1d", "CCI_CrownCastle_vol_20d", "EXC_Exelon_ret_1d", "VRP_ma5"], "is_new": true}, {"model_id": "new_h5_CALM_LightGBM_N30_t4", "algo": "LightGBM", "regime": "CALM", "horizon": 5, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "FedFunds_zscore_60d", "Nikkei_Japan_vol_20d", "ENB_EnbridgeInc_ret_1d", "DAX_Germany_zscore_60d", "EFFR_ret_1d", "TM_Telephone_ret_1d", "HangSeng_HK_ret_1d", "PFE_ret_1d", "Michigan_Sentiment_ret_20d", "MS_MorganStanley_ret_1d", "ASX_Australia_vol_20d", "BDX_Becton_Dickinson_ret_20d", "XLK_Tech_zscore_60d", "SBUX_vol_20d", "HangSeng_HK_vol_20d", "US5Y_Rate_ret_5d", "3M_ret_5d", "XOM_ret_1d", "CMCSA_ret_1d", "spx_abs_ret_max_5d", "MSTR_Bitcoin3_ret_20d", "CPB_CampbellSoup_zscore_60d", "EWM_Malaysia_ret_1d", "MRK_Merck_zscore_60d", "US1Y_Rate_ret_5d", "CPB_CampbellSoup_ret_5d", "MO_AltriaMG_ret_1d", "US3Y_Rate_ret_5d"], "is_new": true}, {"model_id": "new_h5_CALM_LightGBM_N30_t5", "algo": "LightGBM", "regime": "CALM", "horizon": 5, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "BDX_Becton_Dickinson_ret_20d", "vix_mean_abs_ret_5d", "IBEX_Spain_ret_20d", "SLB_Schlumberger_ret_5d", "XLB_Materials_zscore_60d", "M_Macys_vol_20d", "IYR_US_REIT2_zscore_60d", "ORCL_vol_20d", "3M_vol_20d", "US1Y_Rate_ret_20d", "EWY_Korea_zscore_60d", "MSTR_Bitcoin3_ret_5d", "US3M_Rate_vol_20d", "MO_AltriaMG_ret_1d", "GE_ret_1d", "XLV_Health_zscore_60d", "VVIX_ret_20d", "INTC_ret_1d", "MSTR_Bitcoin3_ret_1d", "XOM_ret_1d", "PLD_Prologis_ret_5d", "PG_ret_20d", "LOW_Lowes_ret_5d", "XLY_Disc_vol_20d", "Nikkei_Japan_vol_20d", "BTI_BritishAmerican_ret_5d", "EMR_Emerson_ret_20d", "EWS_Singapore_ret_5d"], "is_new": true}, {"model_id": "new_h5_CALM_LightGBM_N30_t6", "algo": "LightGBM", "regime": "CALM", "horizon": 5, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "IWM_SmallCap_vol_20d", "DE_Deere_vol_20d", "EQIX_Equinix_ret_5d", "XLK_Tech_zscore_60d", "AMZN_ret_5d", "HUM_Humana_ret_5d", "spx_abs_ret_max_5d", "gjr_condvar_h1", "CLX_Clorox_vol_20d", "LOW_Lowes_ret_20d", "heston_var_ev_h7", "SBUX_ret_5d", "Nikkei_Japan_zscore_60d", "ENB_EnbridgeInc_ret_1d", "VRP_ma5", "GE_ret_1d", "EOG_EOGResources_ret_5d", "AMT_AmericanTower_ret_1d", "CPB_CampbellSoup_ret_5d", "DIS_vol_20d", "CPB_CampbellSoup_zscore_60d", "PFE_ret_1d", "vix_acceleration_1d", "ES_Evergy_ret_1d", "AVB_AvalonBay_zscore_60d", "HangSeng_HK_ret_1d", "EWL_Switzerland_zscore_60d", "EXC_Exelon_ret_1d"], "is_new": true}, {"model_id": "new_h5_CALM_LightGBM_N30_t7", "algo": "LightGBM", "regime": "CALM", "horizon": 5, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "HangSeng_HK_ret_1d", "CI_Cigna_vol_20d", "WTI_Oil_FRED_zscore_60d", "T_ret_1d", "spx_abs_ret_max_5d", "LOW_Lowes_ret_5d", "PG_ret_20d", "PCAR_PaccarInc_ret_5d", "SO_SouthernCo_ret_5d", "QQQ_vol_20d", "EWJ_Japan_vol_20d", "HD_ret_5d", "CTAS_Cintas_vol_20d", "EWQ_France_zscore_60d", "PAYX_Paychex_vol_20d", "DOW_Price_zscore_60d", "EWH_HongKong_ret_5d", "AVB_AvalonBay_zscore_60d", "heston_var_ev_h5", "EWM_Malaysia_ret_1d", "INTC_ret_1d", "SCHW_Schwab_ret_5d", "CPB_CampbellSoup_ret_5d", "gjr_condvar_h1", "MS_MorganStanley_ret_1d", "BTI_BritishAmerican_ret_5d", "SJM_JM_Smucker_ret_5d", "PAYX_Paychex_ret_20d"], "is_new": true}, {"model_id": "new_h5_CALM_GradientBoosting_N5_t0", "algo": "GradientBoosting", "regime": "CALM", "horizon": 5, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWH_HongKong_ret_5d", "LUV_SouthwestAir_ret_5d", "EMR_Emerson_ret_20d"], "is_new": true}, {"model_id": "new_h5_CALM_GradientBoosting_N5_t1", "algo": "GradientBoosting", "regime": "CALM", "horizon": 5, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "ITT_ITTInc_ret_5d", "heston_ev_h3", "XLY_Disc_vol_20d"], "is_new": true}, {"model_id": "new_h5_CALM_GradientBoosting_N5_t2", "algo": "GradientBoosting", "regime": "CALM", "horizon": 5, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "GE_ret_1d", "CPB_CampbellSoup_vol_20d", "HD_zscore_60d"], "is_new": true}, {"model_id": "new_h5_CALM_GradientBoosting_N5_t3", "algo": "GradientBoosting", "regime": "CALM", "horizon": 5, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWQ_France_zscore_60d", "T_ret_1d", "SCHW_Schwab_ret_5d"], "is_new": true}, {"model_id": "new_h5_CALM_GradientBoosting_N5_t4", "algo": "GradientBoosting", "regime": "CALM", "horizon": 5, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "TM_Telephone_vol_20d", "SO_SouthernCo_ret_5d", "HUM_Humana_ret_5d"], "is_new": true}, {"model_id": "new_h5_CALM_GradientBoosting_N5_t5", "algo": "GradientBoosting", "regime": "CALM", "horizon": 5, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "FedFunds_zscore_60d", "hmm_p_stress", "Core_PCE_zscore_60d"], "is_new": true}, {"model_id": "new_h5_CALM_GradientBoosting_N5_t6", "algo": "GradientBoosting", "regime": "CALM", "horizon": 5, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "AXP_Amex_vol_20d", "US3M_Rate_zscore_60d", "MSTR_Bitcoin3_ret_20d"], "is_new": true}, {"model_id": "new_h5_CALM_GradientBoosting_N5_t7", "algo": "GradientBoosting", "regime": "CALM", "horizon": 5, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "IYM_BasicMaterials_ret_20d", "SBUX_vol_20d", "US3Y_Rate_ret_5d"], "is_new": true}, {"model_id": "new_h5_CALM_GradientBoosting_N8_t0", "algo": "GradientBoosting", "regime": "CALM", "horizon": 5, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "LOW_Lowes_ret_20d", "AMGN_Amgen_ret_1d", "TED_Spread_zscore_60d", "TXN_vol_20d", "3M_ret_5d", "Nikkei_Japan_vol_20d"], "is_new": true}, {"model_id": "new_h5_CALM_GradientBoosting_N8_t1", "algo": "GradientBoosting", "regime": "CALM", "horizon": 5, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "GE_ret_1d", "LLY_zscore_60d", "3M_vol_20d", "XOM_ret_20d", "ITT_ITTInc_ret_5d", "spx_vol_5d"], "is_new": true}, {"model_id": "new_h5_CALM_GradientBoosting_N8_t2", "algo": "GradientBoosting", "regime": "CALM", "horizon": 5, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWQ_France_zscore_60d", "SBUX_ret_5d", "AXP_Amex_ret_20d", "EXC_Exelon_ret_1d", "QQQ_vol_20d", "NFCI_ret_5d"], "is_new": true}, {"model_id": "new_h5_CALM_GradientBoosting_N8_t3", "algo": "GradientBoosting", "regime": "CALM", "horizon": 5, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "T_ret_1d", "EWQ_France_ret_20d", "INTC_ret_1d", "BA_ret_1d", "LMT_LockheedMartin_vol_20d", "CPB_CampbellSoup_ret_5d"], "is_new": true}, {"model_id": "new_h5_CALM_GradientBoosting_N8_t4", "algo": "GradientBoosting", "regime": "CALM", "horizon": 5, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "AORD_AUS_zscore_60d", "EQR_Equity_ret_1d", "PFE_ret_1d", "LOW_Lowes_ret_5d", "FedFunds_zscore_60d", "heston_ev_h3"], "is_new": true}, {"model_id": "new_h5_CALM_GradientBoosting_N8_t5", "algo": "GradientBoosting", "regime": "CALM", "horizon": 5, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EQIX_Equinix_ret_5d", "VOD_Vodafone_zscore_60d", "Industrial_Production_zscore_60d", "AMT_AmericanTower_ret_1d", "EWQ_France_zscore_60d", "gjr_condvar_h1"], "is_new": true}, {"model_id": "new_h5_CALM_GradientBoosting_N8_t6", "algo": "GradientBoosting", "regime": "CALM", "horizon": 5, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "TXN_vol_20d", "EWM_Malaysia_ret_1d", "HangSeng_HK_ret_1d", "ASX_Australia_vol_20d", "EFFR_ret_1d", "CPB_CampbellSoup_vol_20d"], "is_new": true}, {"model_id": "new_h5_CALM_GradientBoosting_N8_t7", "algo": "GradientBoosting", "regime": "CALM", "horizon": 5, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "HangSeng_HK_ret_1d", "US3Y_Rate_ret_5d", "T10Y2Y_Spread_ret_5d", "AVB_AvalonBay_zscore_60d", "XOM_ret_20d", "gjr_condvar_h1"], "is_new": true}, {"model_id": "new_h5_CALM_GradientBoosting_N10_t0", "algo": "GradientBoosting", "regime": "CALM", "horizon": 5, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "LOW_Lowes_ret_5d", "EWA_Australia_ret_1d", "HangSeng_HK_vol_20d", "heston_ev_h3", "US6M_Rate_ret_20d", "EWJ_Japan_vol_20d", "AMT_AmericanTower_ret_1d", "XLY_Disc_vol_20d"], "is_new": true}, {"model_id": "new_h5_CALM_GradientBoosting_N10_t1", "algo": "GradientBoosting", "regime": "CALM", "horizon": 5, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "IBEX_Spain_ret_20d", "XLB_Materials_zscore_60d", "DAX_Germany_vol_20d", "NOC_Northrop_ret_20d", "MSTR_Bitcoin3_ret_1d", "BA_ret_1d", "MRK_Merck_zscore_60d", "ITT_ITTInc_ret_5d"], "is_new": true}, {"model_id": "new_h5_CALM_GradientBoosting_N10_t2", "algo": "GradientBoosting", "regime": "CALM", "horizon": 5, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EXC_Exelon_zscore_60d", "LOW_Lowes_ret_20d", "Industrial_Production_zscore_60d", "heston_var_ev_h3", "HUM_Humana_ret_5d", "EWA_Australia_zscore_60d", "NOC_Northrop_ret_20d", "ES_Evergy_ret_1d"], "is_new": true}, {"model_id": "new_h5_CALM_GradientBoosting_N10_t3", "algo": "GradientBoosting", "regime": "CALM", "horizon": 5, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "Brent_Oil_FRED_ret_20d", "EFFR_ret_1d", "Brent_Oil_FRED_ret_5d", "AXP_Amex_ret_20d", "US1Y_Rate_ret_5d", "DAX_Germany_vol_20d", "Core_PCE_zscore_60d", "AVB_AvalonBay_zscore_60d"], "is_new": true}, {"model_id": "new_h5_CALM_GradientBoosting_N10_t4", "algo": "GradientBoosting", "regime": "CALM", "horizon": 5, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "T_ret_1d", "AMZN_ret_5d", "ASX_Australia_ret_5d", "PAYX_Paychex_vol_20d", "SJM_JM_Smucker_ret_1d", "US30Y_Rate_ret_20d", "WTI_Oil_FRED_zscore_60d", "SPY_zscore_60d"], "is_new": true}, {"model_id": "new_h5_CALM_GradientBoosting_N10_t5", "algo": "GradientBoosting", "regime": "CALM", "horizon": 5, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "NOC_Northrop_ret_20d", "MSTR_Bitcoin3_ret_5d", "Nikkei_Japan_zscore_60d", "SLB_Schlumberger_ret_5d", "spx_vol_5d", "SLB_Schlumberger_ret_1d", "HangSeng_HK_ret_1d", "EWG_Germany_ret_20d"], "is_new": true}, {"model_id": "new_h5_CALM_GradientBoosting_N10_t6", "algo": "GradientBoosting", "regime": "CALM", "horizon": 5, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "ASX_Australia_ret_5d", "VVIX_ret_20d", "TM_Telephone_vol_20d", "EWJ_Japan_vol_20d", "EWM_Malaysia_ret_1d", "TGT_Target_zscore_60d", "BTI_BritishAmerican_ret_20d", "spx_vol_5d"], "is_new": true}, {"model_id": "new_h5_CALM_GradientBoosting_N10_t7", "algo": "GradientBoosting", "regime": "CALM", "horizon": 5, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "XLF_Fin_vol_20d", "AXP_Amex_ret_20d", "CPB_CampbellSoup_zscore_60d", "GD_GeneralDynamics_zscore_60d", "BA_ret_1d", "LOW_Lowes_ret_5d", "HangSeng_HK_vol_20d", "EWG_Germany_ret_20d"], "is_new": true}, {"model_id": "new_h5_CALM_GradientBoosting_N12_t0", "algo": "GradientBoosting", "regime": "CALM", "horizon": 5, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "XOM_ret_1d", "SPY_zscore_60d", "WTI_Oil_FRED_zscore_60d", "T10Y2Y_Spread_ret_5d", "FedFunds_zscore_60d", "Retail_Sales_zscore_60d", "spx_momentum_3d", "EWS_Singapore_ret_5d", "SO_SouthernCo_ret_5d", "HD_ret_20d"], "is_new": true}, {"model_id": "new_h5_CALM_GradientBoosting_N12_t1", "algo": "GradientBoosting", "regime": "CALM", "horizon": 5, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EQIX_Equinix_ret_5d", "MO_AltriaMG_ret_1d", "SPY_zscore_60d", "AMD_ret_5d", "CMCSA_ret_1d", "Industrial_Production_zscore_60d", "EWM_Malaysia_zscore_60d", "vix_acceleration_1d", "T10Y2Y_Spread_ret_5d", "HD_ret_20d"], "is_new": true}, {"model_id": "new_h5_CALM_GradientBoosting_N12_t2", "algo": "GradientBoosting", "regime": "CALM", "horizon": 5, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "ASX_Australia_vol_20d", "EWG_Germany_vol_20d", "EQR_Equity_ret_1d", "EWM_Malaysia_ret_1d", "EMR_Emerson_ret_20d", "AMD_ret_1d", "CTAS_Cintas_vol_20d", "XLF_Fin_vol_20d", "DE_Deere_vol_20d", "MSTR_Bitcoin3_ret_20d"], "is_new": true}, {"model_id": "new_h5_CALM_GradientBoosting_N12_t3", "algo": "GradientBoosting", "regime": "CALM", "horizon": 5, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "SBUX_zscore_60d", "EWM_Malaysia_vol_20d", "SLB_Schlumberger_ret_1d", "LMT_LockheedMartin_vol_20d", "ITT_ITTInc_ret_5d", "LUV_SouthwestAir_ret_5d", "AXP_Amex_vol_20d", "LLY_zscore_60d", "CPB_CampbellSoup_ret_20d", "XOM_ret_20d"], "is_new": true}, {"model_id": "new_h5_CALM_GradientBoosting_N12_t4", "algo": "GradientBoosting", "regime": "CALM", "horizon": 5, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "CMCSA_ret_1d", "HangSeng_HK_ret_5d", "EFFR_vol_20d", "MSTR_Bitcoin3_ret_5d", "NVDA_vol_20d", "DHR_ret_1d", "HUM_Humana_ret_5d", "HangSeng_HK_vol_20d", "EWM_Malaysia_vol_20d", "NFCI_ret_5d"], "is_new": true}, {"model_id": "new_h5_CALM_GradientBoosting_N12_t5", "algo": "GradientBoosting", "regime": "CALM", "horizon": 5, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "gjr_condvar_h1", "CLX_Clorox_vol_20d", "AMGN_Amgen_ret_1d", "SJM_JM_Smucker_ret_5d", "AVB_AvalonBay_zscore_60d", "BA_ret_1d", "EQIX_Equinix_ret_5d", "heston_var_ev_h5", "MS_MorganStanley_ret_5d", "GILD_Gilead_ret_20d"], "is_new": true}, {"model_id": "new_h5_CALM_GradientBoosting_N12_t6", "algo": "GradientBoosting", "regime": "CALM", "horizon": 5, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "spx_momentum_3d", "VOD_Vodafone_zscore_60d", "Retail_Sales_zscore_60d", "EXC_Exelon_zscore_60d", "TED_Spread_vol_20d", "VRP_ma5", "LOW_Lowes_ret_20d", "Michigan_Sentiment_ret_20d", "GD_GeneralDynamics_zscore_60d", "DHR_ret_1d"], "is_new": true}, {"model_id": "new_h5_CALM_GradientBoosting_N12_t7", "algo": "GradientBoosting", "regime": "CALM", "horizon": 5, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWJ_Japan_vol_20d", "CMCSA_ret_1d", "PAYX_Paychex_ret_20d", "SBUX_zscore_60d", "US7Y_Rate_ret_20d", "BTI_BritishAmerican_ret_5d", "3M_vol_20d", "LOW_Lowes_ret_5d", "EWM_Malaysia_zscore_60d", "EOG_EOGResources_vol_20d"], "is_new": true}, {"model_id": "new_h5_CALM_GradientBoosting_N15_t0", "algo": "GradientBoosting", "regime": "CALM", "horizon": 5, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWM_Malaysia_ret_1d", "EWH_HongKong_ret_5d", "SJM_JM_Smucker_ret_1d", "EWC_Canada_zscore_60d", "EXC_Exelon_ret_1d", "EWA_Australia_zscore_60d", "HangSeng_HK_vol_20d", "TXN_vol_20d", "US1Y_Rate_ret_20d", "XLF_Fin_vol_20d", "Brent_Oil_FRED_ret_20d", "XLY_Disc_vol_20d", "BLK_BlackRock_zscore_60d"], "is_new": true}, {"model_id": "new_h5_CALM_GradientBoosting_N15_t1", "algo": "GradientBoosting", "regime": "CALM", "horizon": 5, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWY_Korea_zscore_60d", "EQIX_Equinix_ret_5d", "AMT_AmericanTower_ret_1d", "Brent_Oil_FRED_ret_20d", "DE_Deere_ret_5d", "PAYX_Paychex_zscore_60d", "PG_ret_20d", "MSTR_Bitcoin3_ret_1d", "AORD_AUS_zscore_60d", "MS_MorganStanley_ret_1d", "MS_MorganStanley_zscore_60d", "DE_Deere_vol_20d", "MRK_Merck_zscore_60d"], "is_new": true}, {"model_id": "new_h5_CALM_GradientBoosting_N15_t2", "algo": "GradientBoosting", "regime": "CALM", "horizon": 5, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "HUM_Humana_ret_5d", "HangSeng_HK_ret_1d", "CTAS_Cintas_vol_20d", "CPB_CampbellSoup_ret_20d", "DE_Deere_ret_5d", "SJM_JM_Smucker_ret_5d", "IBEX_Spain_ret_20d", "HD_ret_5d", "EOG_EOGResources_vol_20d", "CMCSA_ret_1d", "SBUX_ret_5d", "SBUX_vol_20d", "EWY_Korea_zscore_60d"], "is_new": true}, {"model_id": "new_h5_CALM_GradientBoosting_N15_t3", "algo": "GradientBoosting", "regime": "CALM", "horizon": 5, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "BDX_Becton_Dickinson_ret_20d", "spx_momentum_3d", "HD_ret_5d", "QQQ_vol_20d", "US1Y_Rate_ret_5d", "US3M_Rate_vol_20d", "vix_mean_abs_ret_5d", "CLX_Clorox_vol_20d", "EWG_Germany_ret_20d", "Nikkei_Japan_zscore_60d", "Retail_Sales_zscore_60d", "CTAS_Cintas_vol_20d", "ORCL_vol_20d"], "is_new": true}, {"model_id": "new_h5_CALM_GradientBoosting_N15_t4", "algo": "GradientBoosting", "regime": "CALM", "horizon": 5, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "heston_ev_h3", "EFFR_ret_1d", "MSTR_Bitcoin3_ret_5d", "EWQ_France_ret_20d", "TED_Spread_zscore_60d", "DAX_Germany_zscore_60d", "PAYX_Paychex_zscore_60d", "VVIX_ret_20d", "HangSeng_HK_vol_20d", "CLX_Clorox_vol_20d", "LMT_LockheedMartin_ret_1d", "heston_var_ev_h7", "LLY_zscore_60d"], "is_new": true}, {"model_id": "new_h5_CALM_GradientBoosting_N15_t5", "algo": "GradientBoosting", "regime": "CALM", "horizon": 5, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWC_Canada_zscore_60d", "AXP_Amex_ret_20d", "XLB_Materials_zscore_60d", "T10Y2Y_Spread_ret_5d", "Retail_Sales_zscore_60d", "EWL_Switzerland_zscore_60d", "SO_SouthernCo_ret_5d", "EMR_Emerson_ret_20d", "GILD_Gilead_ret_20d", "EOG_EOGResources_ret_5d", "VRP_ma5", "TXN_vol_20d", "LOW_Lowes_ret_5d"], "is_new": true}, {"model_id": "new_h5_CALM_GradientBoosting_N15_t6", "algo": "GradientBoosting", "regime": "CALM", "horizon": 5, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "HD_zscore_60d", "XLV_Health_zscore_60d", "PAYX_Paychex_vol_20d", "XOM_ret_20d", "vix_acceleration_1d", "EWM_Malaysia_zscore_60d", "HD_ret_1d", "HangSeng_HK_vol_20d", "MS_MorganStanley_ret_1d", "EWH_HongKong_ret_5d", "spx_momentum_3d", "HUM_Humana_ret_5d", "MSTR_Bitcoin3_ret_5d"], "is_new": true}, {"model_id": "new_h5_CALM_GradientBoosting_N15_t7", "algo": "GradientBoosting", "regime": "CALM", "horizon": 5, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "Industrial_Production_zscore_60d", "US3M_Rate_zscore_60d", "DAX_Germany_vol_20d", "EWY_Korea_ret_20d", "CPB_CampbellSoup_ret_5d", "CI_Cigna_vol_20d", "AMD_ret_5d", "Brent_Oil_FRED_ret_20d", "MRK_Merck_zscore_60d", "DAX_Germany_zscore_60d", "ENB_EnbridgeInc_ret_1d", "XOM_ret_20d", "TED_Spread_zscore_60d"], "is_new": true}, {"model_id": "new_h5_CALM_GradientBoosting_N20_t0", "algo": "GradientBoosting", "regime": "CALM", "horizon": 5, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EMR_Emerson_ret_20d", "EWC_Canada_zscore_60d", "NWL_Newell_ret_20d", "US30Y_Rate_ret_20d", "TED_Spread_vol_20d", "NFCI_ret_5d", "MSTR_Bitcoin3_ret_5d", "ASX_Australia_ret_5d", "SLB_Schlumberger_ret_5d", "spx_momentum_3d", "US1Y_Rate_ret_20d", "EOG_EOGResources_vol_20d", "EWM_Malaysia_zscore_60d", "DIS_vol_20d", "EWQ_France_ret_20d", "MRK_Merck_zscore_60d", "CPB_CampbellSoup_ret_5d", "Brent_Oil_FRED_ret_20d"], "is_new": true}, {"model_id": "new_h5_CALM_GradientBoosting_N20_t1", "algo": "GradientBoosting", "regime": "CALM", "horizon": 5, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "ITT_ITTInc_ret_5d", "FedFunds_zscore_60d", "US5Y_Rate_ret_5d", "HD_ret_5d", "LOW_Lowes_ret_5d", "PAYX_Paychex_zscore_60d", "AMGN_Amgen_ret_1d", "EWH_HongKong_ret_5d", "CCI_CrownCastle_vol_20d", "PPL_PPL_ret_1d", "M_Macys_vol_20d", "TXN_vol_20d", "spx_momentum_3d", "CPB_CampbellSoup_vol_20d", "BDX_Becton_Dickinson_ret_20d", "CLX_Clorox_vol_20d", "spx_abs_ret_max_5d", "EWJ_Japan_vol_20d"], "is_new": true}, {"model_id": "new_h5_CALM_GradientBoosting_N20_t2", "algo": "GradientBoosting", "regime": "CALM", "horizon": 5, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "SBUX_ret_5d", "EWA_Australia_ret_1d", "BDX_Becton_Dickinson_ret_20d", "SLB_Schlumberger_ret_5d", "CPB_CampbellSoup_ret_5d", "HD_zscore_60d", "CCI_CrownCastle_vol_20d", "HD_ret_1d", "EQR_Equity_ret_1d", "EWM_Malaysia_ret_1d", "ORCL_zscore_60d", "LOW_Lowes_ret_20d", "GILD_Gilead_ret_20d", "JNJ_ret_1d", "MRK_Merck_zscore_60d", "hmm_p_stress", "DAX_Germany_vol_20d", "PAYX_Paychex_ret_20d"], "is_new": true}, {"model_id": "new_h5_CALM_GradientBoosting_N20_t3", "algo": "GradientBoosting", "regime": "CALM", "horizon": 5, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "NEE_NextEra_ret_20d", "Industrial_Production_zscore_60d", "FedFunds_zscore_60d", "SLB_Schlumberger_ret_1d", "PCAR_PaccarInc_ret_5d", "MS_MorganStanley_ret_5d", "MO_AltriaMG_ret_1d", "PAYX_Paychex_vol_20d", "TED_Spread_zscore_60d", "CMCSA_ret_1d", "MRK_Merck_zscore_60d", "US3M_Rate_vol_20d", "Brent_Oil_FRED_ret_20d", "NVDA_vol_20d", "LMT_LockheedMartin_vol_20d", "IYR_US_REIT2_zscore_60d", "heston_ev_h3", "ORCL_zscore_60d"], "is_new": true}, {"model_id": "new_h5_CALM_GradientBoosting_N20_t4", "algo": "GradientBoosting", "regime": "CALM", "horizon": 5, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "Brent_Oil_FRED_ret_5d", "EWL_Switzerland_vol_20d", "ORCL_vol_20d", "BA_ret_1d", "Core_CPI_zscore_60d", "Retail_Sales_zscore_60d", "CMCSA_ret_1d", "HangSeng_HK_ret_5d", "SBUX_ret_5d", "EFFR_vol_20d", "EOG_EOGResources_vol_20d", "DAX_Germany_zscore_60d", "XLF_Fin_vol_20d", "HD_ret_20d", "IYM_BasicMaterials_ret_20d", "EWG_Germany_ret_20d", "CPB_CampbellSoup_ret_5d", "US3M_Rate_vol_20d"], "is_new": true}, {"model_id": "new_h5_CALM_GradientBoosting_N20_t5", "algo": "GradientBoosting", "regime": "CALM", "horizon": 5, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "HD_ret_20d", "HD_zscore_60d", "CPB_CampbellSoup_ret_5d", "T_ret_1d", "spx_vol_5d", "AMD_ret_5d", "IYR_US_REIT2_zscore_60d", "GD_GeneralDynamics_zscore_60d", "AMD_ret_1d", "AORD_AUS_zscore_60d", "SPY_zscore_60d", "IWM_SmallCap_vol_20d", "EWM_Malaysia_ret_1d", "vix_acceleration_1d", "XLB_Materials_zscore_60d", "GE_ret_1d", "EFFR_vol_20d", "AMGN_Amgen_ret_1d"], "is_new": true}, {"model_id": "new_h5_CALM_GradientBoosting_N20_t6", "algo": "GradientBoosting", "regime": "CALM", "horizon": 5, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "gjr_condvar_h1", "XLV_Health_zscore_60d", "PG_ret_20d", "VRP_ma5", "TM_Telephone_vol_20d", "Nikkei_Japan_vol_20d", "IYM_BasicMaterials_ret_20d", "US5Y_Rate_ret_5d", "CCI_CrownCastle_vol_20d", "EWM_Malaysia_ret_1d", "CPB_CampbellSoup_ret_5d", "CPB_CampbellSoup_vol_20d", "US1Y_Rate_ret_5d", "SJM_JM_Smucker_ret_1d", "AMT_AmericanTower_ret_1d", "spx_vol_5d", "T10Y2Y_Spread_ret_5d", "PFE_ret_1d"], "is_new": true}, {"model_id": "new_h5_CALM_GradientBoosting_N20_t7", "algo": "GradientBoosting", "regime": "CALM", "horizon": 5, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EXC_Exelon_zscore_60d", "US7Y_Rate_ret_20d", "Core_PCE_zscore_60d", "NWL_Newell_ret_20d", "SBUX_vol_20d", "BTI_BritishAmerican_ret_5d", "ITT_ITTInc_ret_5d", "Nikkei_Japan_zscore_60d", "PPL_PPL_ret_1d", "AXP_Amex_vol_20d", "US5Y_Rate_ret_5d", "TED_Spread_vol_20d", "SCHW_Schwab_ret_5d", "TXN_vol_20d", "TGT_Target_zscore_60d", "IYM_BasicMaterials_ret_20d", "T_ret_1d", "EQIX_Equinix_ret_5d"], "is_new": true}, {"model_id": "new_h5_CALM_GradientBoosting_N25_t0", "algo": "GradientBoosting", "regime": "CALM", "horizon": 5, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "spx_abs_ret_max_5d", "SPY_zscore_60d", "LUV_SouthwestAir_ret_5d", "PLD_Prologis_ret_5d", "XLK_Tech_zscore_60d", "HangSeng_HK_ret_1d", "BLK_BlackRock_zscore_60d", "BTI_BritishAmerican_ret_20d", "Industrial_Production_zscore_60d", "HD_ret_5d", "CI_Cigna_vol_20d", "MO_AltriaMG_ret_1d", "AMGN_Amgen_ret_1d", "M_Macys_vol_20d", "SO_SouthernCo_ret_5d", "US7Y_Rate_ret_20d", "US1Y_Rate_ret_5d", "JNJ_ret_1d", "CLX_Clorox_vol_20d", "CPB_CampbellSoup_zscore_60d", "TXN_vol_20d", "US1Y_Rate_ret_20d", "CMCSA_ret_1d"], "is_new": true}, {"model_id": "new_h5_CALM_GradientBoosting_N25_t1", "algo": "GradientBoosting", "regime": "CALM", "horizon": 5, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWG_Germany_ret_20d", "GE_ret_1d", "SLB_Schlumberger_ret_5d", "BDX_Becton_Dickinson_ret_20d", "US30Y_Rate_ret_20d", "ITT_ITTInc_ret_5d", "Core_CPI_zscore_60d", "EWY_Korea_zscore_60d", "XLV_Health_zscore_60d", "Michigan_Sentiment_ret_20d", "NFCI_ret_5d", "PAYX_Paychex_zscore_60d", "MRK_Merck_zscore_60d", "PFE_ret_1d", "CPB_CampbellSoup_ret_5d", "MSTR_Bitcoin3_ret_20d", "US3Y_Rate_ret_5d", "INTC_ret_5d", "HD_zscore_60d", "AMZN_ret_5d", "DHR_ret_1d", "EWA_Australia_zscore_60d", "CPB_CampbellSoup_ret_20d"], "is_new": true}, {"model_id": "new_h5_CALM_GradientBoosting_N25_t2", "algo": "GradientBoosting", "regime": "CALM", "horizon": 5, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "US30Y_Rate_ret_20d", "EWC_Canada_zscore_60d", "XLK_Tech_zscore_60d", "spx_abs_ret_max_5d", "MSTR_Bitcoin3_ret_20d", "VOD_Vodafone_zscore_60d", "XLY_Disc_vol_20d", "LLY_zscore_60d", "PG_ret_20d", "CPB_CampbellSoup_ret_20d", "EWL_Switzerland_zscore_60d", "spx_vol_5d", "LMT_LockheedMartin_vol_20d", "heston_var_ev_h3", "AMT_AmericanTower_ret_1d", "AXP_Amex_ret_20d", "CPB_CampbellSoup_zscore_60d", "NWL_Newell_ret_20d", "T_ret_1d", "HD_zscore_60d", "heston_var_ev_h5", "BTI_BritishAmerican_ret_20d", "DIS_vol_20d"], "is_new": true}, {"model_id": "new_h5_CALM_GradientBoosting_N25_t3", "algo": "GradientBoosting", "regime": "CALM", "horizon": 5, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWL_Switzerland_zscore_60d", "SPY_zscore_60d", "PAYX_Paychex_ret_20d", "TM_Telephone_ret_1d", "XLV_Health_zscore_60d", "EWA_Australia_ret_1d", "AXP_Amex_ret_20d", "EWG_Germany_ret_20d", "spx_momentum_3d", "Core_CPI_zscore_60d", "HUM_Humana_ret_5d", "PLD_Prologis_ret_5d", "MS_MorganStanley_ret_5d", "BA_ret_1d", "GILD_Gilead_ret_20d", "3M_ret_5d", "US3Y_Rate_ret_5d", "MS_MorganStanley_zscore_60d", "ES_Evergy_ret_1d", "NWL_Newell_ret_20d", "DHR_vol_20d", "US3M_Rate_vol_20d", "EMR_Emerson_ret_20d"], "is_new": true}, {"model_id": "new_h5_CALM_GradientBoosting_N25_t4", "algo": "GradientBoosting", "regime": "CALM", "horizon": 5, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "PAYX_Paychex_ret_20d", "Nikkei_Japan_vol_20d", "EOG_EOGResources_ret_5d", "Core_PCE_zscore_60d", "3M_vol_20d", "US3M_Rate_vol_20d", "T_ret_1d", "LOW_Lowes_ret_5d", "EWG_Germany_ret_20d", "EQR_Equity_ret_1d", "vix_mean_abs_ret_5d", "DE_Deere_ret_5d", "ORCL_zscore_60d", "DE_Deere_vol_20d", "AVB_AvalonBay_zscore_60d", "Nikkei_Japan_zscore_60d", "MS_MorganStanley_zscore_60d", "PG_ret_20d", "MO_AltriaMG_ret_1d", "GILD_Gilead_ret_20d", "US6M_Rate_ret_20d", "AMD_ret_5d", "LUV_SouthwestAir_ret_5d"], "is_new": true}, {"model_id": "new_h5_CALM_GradientBoosting_N25_t5", "algo": "GradientBoosting", "regime": "CALM", "horizon": 5, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWQ_France_zscore_60d", "PCAR_PaccarInc_ret_5d", "ITT_ITTInc_ret_5d", "vix_mean_abs_ret_5d", "PPL_PPL_ret_1d", "EXC_Exelon_ret_1d", "MRK_Merck_zscore_60d", "BLK_BlackRock_zscore_60d", "SLB_Schlumberger_ret_1d", "SPY_zscore_60d", "EWG_Germany_vol_20d", "3M_vol_20d", "M_Macys_vol_20d", "US30Y_Rate_ret_20d", "DHR_ret_1d", "US3M_Rate_vol_20d", "EXC_Exelon_zscore_60d", "heston_ev_h3", "DAX_Germany_vol_20d", "PAYX_Paychex_ret_20d", "BA_ret_1d", "HD_ret_1d", "US6M_Rate_ret_20d"], "is_new": true}, {"model_id": "new_h5_CALM_GradientBoosting_N25_t6", "algo": "GradientBoosting", "regime": "CALM", "horizon": 5, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "US7Y_Rate_ret_20d", "HangSeng_HK_ret_1d", "DE_Deere_vol_20d", "PAYX_Paychex_vol_20d", "XLY_Disc_vol_20d", "CI_Cigna_vol_20d", "vix_mean_abs_ret_5d", "BTI_BritishAmerican_ret_5d", "Nikkei_Japan_vol_20d", "EWJ_Japan_vol_20d", "PAYX_Paychex_zscore_60d", "XLV_Health_zscore_60d", "T10Y2Y_Spread_ret_5d", "AORD_AUS_zscore_60d", "ASX_Australia_vol_20d", "GD_GeneralDynamics_zscore_60d", "spx_momentum_3d", "DIS_vol_20d", "3M_ret_5d", "ENB_EnbridgeInc_ret_1d", "TED_Spread_zscore_60d", "TM_Telephone_ret_1d", "US1Y_Rate_ret_5d"], "is_new": true}, {"model_id": "new_h5_CALM_GradientBoosting_N25_t7", "algo": "GradientBoosting", "regime": "CALM", "horizon": 5, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "US3M_Rate_vol_20d", "SBUX_vol_20d", "3M_ret_5d", "US30Y_Rate_ret_20d", "ORCL_zscore_60d", "QQQ_vol_20d", "ASX_Australia_vol_20d", "BTI_BritishAmerican_ret_5d", "INTC_ret_1d", "EOG_EOGResources_ret_5d", "HangSeng_HK_ret_1d", "EOG_EOGResources_vol_20d", "GD_GeneralDynamics_zscore_60d", "HD_ret_20d", "EWY_Korea_ret_20d", "CLX_Clorox_vol_20d", "AXP_Amex_ret_20d", "DAX_Germany_zscore_60d", "US3Y_Rate_ret_5d", "EWL_Switzerland_vol_20d", "DHR_vol_20d", "CPB_CampbellSoup_ret_20d", "AMT_AmericanTower_ret_1d"], "is_new": true}, {"model_id": "new_h5_CALM_GradientBoosting_N30_t0", "algo": "GradientBoosting", "regime": "CALM", "horizon": 5, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "ASX_Australia_vol_20d", "vix_mean_abs_ret_5d", "VOD_Vodafone_zscore_60d", "ES_Evergy_ret_1d", "JNJ_ret_1d", "FedFunds_zscore_60d", "DIS_vol_20d", "SJM_JM_Smucker_ret_1d", "AMZN_ret_5d", "spx_vol_5d", "SO_SouthernCo_ret_5d", "EWL_Switzerland_vol_20d", "CPB_CampbellSoup_zscore_60d", "gjr_condvar_h1", "DAX_Germany_vol_20d", "TXN_vol_20d", "BLK_BlackRock_zscore_60d", "EWQ_France_ret_20d", "HD_ret_1d", "EWA_Australia_ret_1d", "AORD_AUS_zscore_60d", "PCAR_PaccarInc_ret_5d", "CPB_CampbellSoup_vol_20d", "HangSeng_HK_ret_5d", "TM_Telephone_vol_20d", "T_ret_1d", "XOM_ret_20d", "AVB_AvalonBay_zscore_60d"], "is_new": true}, {"model_id": "new_h5_CALM_GradientBoosting_N30_t1", "algo": "GradientBoosting", "regime": "CALM", "horizon": 5, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "T_ret_1d", "DE_Deere_ret_5d", "SCHW_Schwab_ret_5d", "heston_var_ev_h3", "LOW_Lowes_ret_20d", "WTI_Oil_FRED_zscore_60d", "TXN_vol_20d", "MS_MorganStanley_zscore_60d", "SPY_zscore_60d", "MO_AltriaMG_ret_1d", "US1Y_Rate_ret_20d", "CPB_CampbellSoup_ret_20d", "VVIX_ret_20d", "spx_vol_5d", "HD_ret_1d", "SLB_Schlumberger_ret_1d", "gjr_condvar_h1", "HangSeng_HK_ret_1d", "NOC_Northrop_ret_20d", "ORCL_zscore_60d", "EXC_Exelon_zscore_60d", "heston_var_ev_h7", "EFFR_ret_1d", "EOG_EOGResources_ret_5d", "EWA_Australia_ret_1d", "US6M_Rate_ret_20d", "TM_Telephone_ret_1d", "M_Macys_vol_20d"], "is_new": true}, {"model_id": "new_h5_CALM_GradientBoosting_N30_t2", "algo": "GradientBoosting", "regime": "CALM", "horizon": 5, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "HangSeng_HK_ret_5d", "NEE_NextEra_ret_20d", "HD_ret_1d", "PG_ret_20d", "CMCSA_ret_1d", "SBUX_vol_20d", "HD_zscore_60d", "DHR_ret_1d", "MS_MorganStanley_ret_1d", "LOW_Lowes_ret_5d", "TXN_vol_20d", "gjr_condvar_h1", "HangSeng_HK_ret_1d", "BA_ret_1d", "vix_acceleration_1d", "ORCL_vol_20d", "CI_Cigna_vol_20d", "DOW_Price_zscore_60d", "GE_ret_1d", "LLY_zscore_60d", "AMZN_ret_5d", "XLF_Fin_vol_20d", "BTI_BritishAmerican_ret_5d", "Nikkei_Japan_zscore_60d", "ASX_Australia_vol_20d", "CPB_CampbellSoup_ret_5d", "EFFR_vol_20d", "WTI_Oil_FRED_zscore_60d"], "is_new": true}, {"model_id": "new_h5_CALM_GradientBoosting_N30_t3", "algo": "GradientBoosting", "regime": "CALM", "horizon": 5, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "NEE_NextEra_ret_20d", "BA_ret_1d", "PCAR_PaccarInc_ret_5d", "MS_MorganStanley_zscore_60d", "EQIX_Equinix_ret_5d", "CPB_CampbellSoup_ret_5d", "AMT_AmericanTower_ret_1d", "US7Y_Rate_ret_20d", "M_Macys_vol_20d", "XLK_Tech_zscore_60d", "EMR_Emerson_ret_20d", "vix_mean_abs_ret_5d", "EQR_Equity_ret_1d", "CMCSA_ret_1d", "MSTR_Bitcoin3_ret_1d", "US30Y_Rate_ret_20d", "GE_ret_1d", "PPL_PPL_ret_1d", "XOM_ret_1d", "AMGN_Amgen_ret_1d", "EFFR_vol_20d", "AXP_Amex_vol_20d", "MSTR_Bitcoin3_ret_20d", "EWA_Australia_zscore_60d", "BTI_BritishAmerican_ret_5d", "IBEX_Spain_ret_20d", "US5Y_Rate_ret_5d", "ASX_Australia_ret_5d"], "is_new": true}, {"model_id": "new_h5_CALM_GradientBoosting_N30_t4", "algo": "GradientBoosting", "regime": "CALM", "horizon": 5, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWJ_Japan_vol_20d", "EQR_Equity_ret_1d", "US7Y_Rate_ret_20d", "LOW_Lowes_ret_20d", "HangSeng_HK_ret_1d", "3M_vol_20d", "LLY_zscore_60d", "ORCL_vol_20d", "TED_Spread_zscore_60d", "MS_MorganStanley_ret_1d", "HD_ret_5d", "DIS_vol_20d", "US3Y_Rate_ret_5d", "MS_MorganStanley_ret_5d", "DAX_Germany_vol_20d", "SBUX_zscore_60d", "EOG_EOGResources_vol_20d", "GILD_Gilead_ret_20d", "PFE_ret_1d", "DHR_ret_1d", "XLK_Tech_zscore_60d", "EWH_HongKong_ret_5d", "BTI_BritishAmerican_ret_5d", "TED_Spread_vol_20d", "US1Y_Rate_ret_5d", "US3M_Rate_zscore_60d", "VRP_ma5", "ASX_Australia_vol_20d"], "is_new": true}, {"model_id": "new_h5_CALM_GradientBoosting_N30_t5", "algo": "GradientBoosting", "regime": "CALM", "horizon": 5, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "MS_MorganStanley_ret_1d", "DHR_vol_20d", "AMGN_Amgen_ret_1d", "EWH_HongKong_ret_5d", "PAYX_Paychex_ret_20d", "HD_ret_20d", "HD_ret_5d", "LMT_LockheedMartin_vol_20d", "QQQ_vol_20d", "Brent_Oil_FRED_ret_20d", "US5Y_Rate_ret_5d", "XOM_ret_1d", "Brent_Oil_FRED_ret_5d", "ORCL_vol_20d", "Retail_Sales_zscore_60d", "EWG_Germany_ret_20d", "hmm_p_stress", "DIS_vol_20d", "heston_ev_h3", "EXC_Exelon_ret_1d", "XLB_Materials_zscore_60d", "CMCSA_ret_1d", "CPB_CampbellSoup_ret_5d", "HD_ret_1d", "AMZN_ret_5d", "PG_ret_20d", "US1Y_Rate_ret_5d", "XOM_ret_20d"], "is_new": true}, {"model_id": "new_h5_CALM_GradientBoosting_N30_t6", "algo": "GradientBoosting", "regime": "CALM", "horizon": 5, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "VVIX_ret_20d", "DE_Deere_ret_5d", "VRP_ma5", "AMD_ret_1d", "SCHW_Schwab_ret_5d", "BDX_Becton_Dickinson_ret_20d", "EWA_Australia_zscore_60d", "SPY_zscore_60d", "EWL_Switzerland_zscore_60d", "US3M_Rate_vol_20d", "HD_zscore_60d", "EOG_EOGResources_vol_20d", "PG_ret_20d", "Core_CPI_zscore_60d", "heston_var_ev_h3", "FedFunds_zscore_60d", "DHR_vol_20d", "HangSeng_HK_ret_1d", "SBUX_zscore_60d", "hmm_p_stress", "NOC_Northrop_ret_20d", "XLB_Materials_zscore_60d", "US3M_Rate_zscore_60d", "DE_Deere_vol_20d", "DAX_Germany_zscore_60d", "US5Y_Rate_ret_5d", "INTC_ret_1d", "PAYX_Paychex_ret_20d"], "is_new": true}, {"model_id": "new_h5_CALM_GradientBoosting_N30_t7", "algo": "GradientBoosting", "regime": "CALM", "horizon": 5, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "AORD_AUS_zscore_60d", "HD_ret_1d", "SJM_JM_Smucker_ret_5d", "US3Y_Rate_ret_5d", "LOW_Lowes_ret_5d", "EMR_Emerson_ret_20d", "FedFunds_zscore_60d", "Core_PCE_zscore_60d", "GD_GeneralDynamics_zscore_60d", "EWL_Switzerland_zscore_60d", "vix_acceleration_1d", "LUV_SouthwestAir_ret_5d", "Nikkei_Japan_vol_20d", "HangSeng_HK_vol_20d", "NEE_NextEra_ret_20d", "MS_MorganStanley_zscore_60d", "EWL_Switzerland_vol_20d", "XLB_Materials_zscore_60d", "SBUX_ret_5d", "CCI_CrownCastle_vol_20d", "spx_vol_5d", "AVB_AvalonBay_zscore_60d", "CMCSA_ret_1d", "DHR_ret_1d", "AXP_Amex_ret_20d", "MSTR_Bitcoin3_ret_5d", "EWH_HongKong_ret_5d", "EWQ_France_zscore_60d"], "is_new": true}, {"model_id": "new_h5_CALM_RandomForest_N5_t0", "algo": "RandomForest", "regime": "CALM", "horizon": 5, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "XLY_Disc_vol_20d", "GE_ret_1d", "VVIX_ret_20d"], "is_new": true}, {"model_id": "new_h5_CALM_RandomForest_N5_t1", "algo": "RandomForest", "regime": "CALM", "horizon": 5, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "LOW_Lowes_ret_20d", "NVDA_vol_20d", "HD_ret_1d"], "is_new": true}, {"model_id": "new_h5_CALM_RandomForest_N5_t2", "algo": "RandomForest", "regime": "CALM", "horizon": 5, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "MS_MorganStanley_zscore_60d", "ORCL_zscore_60d", "EXC_Exelon_ret_1d"], "is_new": true}, {"model_id": "new_h5_CALM_RandomForest_N5_t3", "algo": "RandomForest", "regime": "CALM", "horizon": 5, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "SLB_Schlumberger_ret_5d", "CPB_CampbellSoup_vol_20d", "MS_MorganStanley_ret_5d"], "is_new": true}, {"model_id": "new_h5_CALM_RandomForest_N5_t4", "algo": "RandomForest", "regime": "CALM", "horizon": 5, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "ENB_EnbridgeInc_ret_1d", "HangSeng_HK_ret_5d", "DHR_vol_20d"], "is_new": true}, {"model_id": "new_h5_CALM_RandomForest_N5_t5", "algo": "RandomForest", "regime": "CALM", "horizon": 5, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "LMT_LockheedMartin_vol_20d", "EWG_Germany_ret_20d", "DE_Deere_vol_20d"], "is_new": true}, {"model_id": "new_h5_CALM_RandomForest_N5_t6", "algo": "RandomForest", "regime": "CALM", "horizon": 5, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "Core_PCE_zscore_60d", "HangSeng_HK_vol_20d", "M_Macys_vol_20d"], "is_new": true}, {"model_id": "new_h5_CALM_RandomForest_N5_t7", "algo": "RandomForest", "regime": "CALM", "horizon": 5, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "US3Y_Rate_ret_5d", "SO_SouthernCo_ret_5d", "EWA_Australia_zscore_60d"], "is_new": true}, {"model_id": "new_h5_CALM_RandomForest_N8_t0", "algo": "RandomForest", "regime": "CALM", "horizon": 5, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "SJM_JM_Smucker_ret_5d", "CPB_CampbellSoup_ret_5d", "XLY_Disc_vol_20d", "US30Y_Rate_ret_20d", "EWA_Australia_zscore_60d", "NWL_Newell_ret_20d"], "is_new": true}, {"model_id": "new_h5_CALM_RandomForest_N8_t1", "algo": "RandomForest", "regime": "CALM", "horizon": 5, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "US6M_Rate_ret_20d", "BDX_Becton_Dickinson_ret_20d", "ITT_ITTInc_ret_5d", "VVIX_ret_20d", "vix_acceleration_1d", "CI_Cigna_vol_20d"], "is_new": true}, {"model_id": "new_h5_CALM_RandomForest_N8_t2", "algo": "RandomForest", "regime": "CALM", "horizon": 5, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "LMT_LockheedMartin_vol_20d", "EWG_Germany_vol_20d", "XLY_Disc_vol_20d", "PFE_ret_1d", "US5Y_Rate_ret_5d", "ASX_Australia_ret_5d"], "is_new": true}, {"model_id": "new_h5_CALM_RandomForest_N8_t3", "algo": "RandomForest", "regime": "CALM", "horizon": 5, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EXC_Exelon_ret_1d", "CMCSA_ret_1d", "AMGN_Amgen_ret_1d", "TM_Telephone_vol_20d", "AXP_Amex_vol_20d", "SJM_JM_Smucker_ret_1d"], "is_new": true}, {"model_id": "new_h5_CALM_RandomForest_N8_t4", "algo": "RandomForest", "regime": "CALM", "horizon": 5, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "spx_momentum_3d", "EWJ_Japan_vol_20d", "ITT_ITTInc_ret_5d", "GE_ret_1d", "MRK_Merck_zscore_60d", "CTAS_Cintas_vol_20d"], "is_new": true}, {"model_id": "new_h5_CALM_RandomForest_N8_t5", "algo": "RandomForest", "regime": "CALM", "horizon": 5, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "DAX_Germany_zscore_60d", "XLF_Fin_vol_20d", "HangSeng_HK_ret_1d", "EOG_EOGResources_vol_20d", "DOW_Price_zscore_60d", "AXP_Amex_ret_20d"], "is_new": true}, {"model_id": "new_h5_CALM_RandomForest_N8_t6", "algo": "RandomForest", "regime": "CALM", "horizon": 5, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWA_Australia_ret_1d", "CPB_CampbellSoup_ret_5d", "DIS_vol_20d", "CPB_CampbellSoup_ret_20d", "LOW_Lowes_ret_5d", "Industrial_Production_zscore_60d"], "is_new": true}, {"model_id": "new_h5_CALM_RandomForest_N8_t7", "algo": "RandomForest", "regime": "CALM", "horizon": 5, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "GD_GeneralDynamics_zscore_60d", "BTI_BritishAmerican_ret_5d", "XLF_Fin_vol_20d", "ORCL_zscore_60d", "CTAS_Cintas_vol_20d", "Nikkei_Japan_zscore_60d"], "is_new": true}, {"model_id": "new_h5_CALM_RandomForest_N10_t0", "algo": "RandomForest", "regime": "CALM", "horizon": 5, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "MSTR_Bitcoin3_ret_20d", "AXP_Amex_ret_20d", "Core_PCE_zscore_60d", "TXN_vol_20d", "CLX_Clorox_vol_20d", "CPB_CampbellSoup_zscore_60d", "heston_var_ev_h7", "PPL_PPL_ret_1d"], "is_new": true}, {"model_id": "new_h5_CALM_RandomForest_N10_t1", "algo": "RandomForest", "regime": "CALM", "horizon": 5, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "vix_mean_abs_ret_5d", "EWY_Korea_ret_20d", "DOW_Price_zscore_60d", "LMT_LockheedMartin_vol_20d", "EWG_Germany_ret_20d", "TM_Telephone_ret_1d", "MS_MorganStanley_zscore_60d", "Brent_Oil_FRED_ret_20d"], "is_new": true}, {"model_id": "new_h5_CALM_RandomForest_N10_t2", "algo": "RandomForest", "regime": "CALM", "horizon": 5, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "US6M_Rate_ret_20d", "EFFR_ret_1d", "Brent_Oil_FRED_ret_20d", "spx_abs_ret_max_5d", "NVDA_vol_20d", "Industrial_Production_zscore_60d", "IBEX_Spain_ret_20d", "TXN_vol_20d"], "is_new": true}, {"model_id": "new_h5_CALM_RandomForest_N10_t3", "algo": "RandomForest", "regime": "CALM", "horizon": 5, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "NEE_NextEra_ret_20d", "US1Y_Rate_ret_20d", "SJM_JM_Smucker_ret_1d", "EQIX_Equinix_ret_5d", "MSTR_Bitcoin3_ret_20d", "Core_CPI_zscore_60d", "Core_PCE_zscore_60d", "SCHW_Schwab_ret_5d"], "is_new": true}, {"model_id": "new_h5_CALM_RandomForest_N10_t4", "algo": "RandomForest", "regime": "CALM", "horizon": 5, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "DAX_Germany_vol_20d", "MRK_Merck_zscore_60d", "EWH_HongKong_ret_5d", "TXN_vol_20d", "EXC_Exelon_ret_1d", "LMT_LockheedMartin_ret_1d", "AVB_AvalonBay_zscore_60d", "EWG_Germany_vol_20d"], "is_new": true}, {"model_id": "new_h5_CALM_RandomForest_N10_t5", "algo": "RandomForest", "regime": "CALM", "horizon": 5, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWM_Malaysia_ret_1d", "CLX_Clorox_vol_20d", "EWC_Canada_zscore_60d", "vix_acceleration_1d", "CPB_CampbellSoup_zscore_60d", "DOW_Price_zscore_60d", "Core_PCE_zscore_60d", "EWG_Germany_ret_20d"], "is_new": true}, {"model_id": "new_h5_CALM_RandomForest_N10_t6", "algo": "RandomForest", "regime": "CALM", "horizon": 5, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EXC_Exelon_ret_1d", "vix_acceleration_1d", "MSTR_Bitcoin3_ret_20d", "MS_MorganStanley_ret_1d", "DAX_Germany_zscore_60d", "XLB_Materials_zscore_60d", "CTAS_Cintas_vol_20d", "EWG_Germany_ret_20d"], "is_new": true}, {"model_id": "new_h5_CALM_RandomForest_N10_t7", "algo": "RandomForest", "regime": "CALM", "horizon": 5, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "IYR_US_REIT2_zscore_60d", "EWM_Malaysia_vol_20d", "VRP_ma5", "GILD_Gilead_ret_20d", "EXC_Exelon_zscore_60d", "AMD_ret_5d", "MSTR_Bitcoin3_ret_5d", "3M_ret_5d"], "is_new": true}, {"model_id": "new_h5_CALM_RandomForest_N12_t0", "algo": "RandomForest", "regime": "CALM", "horizon": 5, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "DE_Deere_ret_5d", "AMD_ret_1d", "JNJ_ret_1d", "US1Y_Rate_ret_20d", "GILD_Gilead_ret_20d", "VVIX_ret_20d", "US1Y_Rate_ret_5d", "HD_ret_20d", "EXC_Exelon_ret_1d", "PAYX_Paychex_zscore_60d"], "is_new": true}, {"model_id": "new_h5_CALM_RandomForest_N12_t1", "algo": "RandomForest", "regime": "CALM", "horizon": 5, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "BLK_BlackRock_zscore_60d", "AORD_AUS_zscore_60d", "Retail_Sales_zscore_60d", "HUM_Humana_ret_5d", "SPY_zscore_60d", "CPB_CampbellSoup_zscore_60d", "EWG_Germany_ret_20d", "EWA_Australia_ret_1d", "MO_AltriaMG_ret_1d", "EWQ_France_ret_20d"], "is_new": true}, {"model_id": "new_h5_CALM_RandomForest_N12_t2", "algo": "RandomForest", "regime": "CALM", "horizon": 5, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "BTI_BritishAmerican_ret_20d", "PAYX_Paychex_ret_20d", "LMT_LockheedMartin_ret_1d", "CTAS_Cintas_vol_20d", "SJM_JM_Smucker_ret_5d", "AORD_AUS_zscore_60d", "3M_ret_5d", "PAYX_Paychex_vol_20d", "spx_vol_5d", "MSTR_Bitcoin3_ret_5d"], "is_new": true}, {"model_id": "new_h5_CALM_RandomForest_N12_t3", "algo": "RandomForest", "regime": "CALM", "horizon": 5, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EXC_Exelon_zscore_60d", "SLB_Schlumberger_ret_5d", "EWQ_France_zscore_60d", "SLB_Schlumberger_ret_1d", "Nikkei_Japan_vol_20d", "US1Y_Rate_ret_5d", "MSTR_Bitcoin3_ret_1d", "CLX_Clorox_vol_20d", "DHR_vol_20d", "EWC_Canada_zscore_60d"], "is_new": true}, {"model_id": "new_h5_CALM_RandomForest_N12_t4", "algo": "RandomForest", "regime": "CALM", "horizon": 5, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "Industrial_Production_zscore_60d", "3M_vol_20d", "Retail_Sales_zscore_60d", "EWG_Germany_vol_20d", "AVB_AvalonBay_zscore_60d", "VVIX_ret_20d", "BLK_BlackRock_zscore_60d", "SBUX_ret_5d", "DAX_Germany_zscore_60d", "US3M_Rate_vol_20d"], "is_new": true}, {"model_id": "new_h5_CALM_RandomForest_N12_t5", "algo": "RandomForest", "regime": "CALM", "horizon": 5, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "MS_MorganStanley_ret_1d", "Nikkei_Japan_vol_20d", "Michigan_Sentiment_ret_20d", "PAYX_Paychex_ret_20d", "EWA_Australia_zscore_60d", "CPB_CampbellSoup_ret_5d", "EWJ_Japan_vol_20d", "XLB_Materials_zscore_60d", "EQIX_Equinix_ret_5d", "SPY_zscore_60d"], "is_new": true}, {"model_id": "new_h5_CALM_RandomForest_N12_t6", "algo": "RandomForest", "regime": "CALM", "horizon": 5, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "IYR_US_REIT2_zscore_60d", "HangSeng_HK_vol_20d", "SBUX_zscore_60d", "AMT_AmericanTower_ret_1d", "Core_PCE_zscore_60d", "vix_mean_abs_ret_5d", "CCI_CrownCastle_vol_20d", "EQR_Equity_ret_1d", "CI_Cigna_vol_20d", "HD_ret_5d"], "is_new": true}, {"model_id": "new_h5_CALM_RandomForest_N12_t7", "algo": "RandomForest", "regime": "CALM", "horizon": 5, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "US3M_Rate_zscore_60d", "EXC_Exelon_zscore_60d", "EMR_Emerson_ret_20d", "PAYX_Paychex_ret_20d", "EOG_EOGResources_ret_5d", "SO_SouthernCo_ret_5d", "DOW_Price_zscore_60d", "TGT_Target_zscore_60d", "SLB_Schlumberger_ret_5d", "Core_PCE_zscore_60d"], "is_new": true}, {"model_id": "new_h5_CALM_RandomForest_N15_t0", "algo": "RandomForest", "regime": "CALM", "horizon": 5, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "AMD_ret_5d", "HUM_Humana_ret_5d", "BA_ret_1d", "EOG_EOGResources_vol_20d", "US1Y_Rate_ret_5d", "spx_momentum_3d", "PG_ret_20d", "HD_ret_20d", "HangSeng_HK_vol_20d", "PFE_ret_1d", "AMGN_Amgen_ret_1d", "MSTR_Bitcoin3_ret_5d", "HD_ret_1d"], "is_new": true}, {"model_id": "new_h5_CALM_RandomForest_N15_t1", "algo": "RandomForest", "regime": "CALM", "horizon": 5, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "Industrial_Production_zscore_60d", "SJM_JM_Smucker_ret_1d", "T10Y2Y_Spread_ret_5d", "EFFR_vol_20d", "XLY_Disc_vol_20d", "VOD_Vodafone_zscore_60d", "EFFR_ret_1d", "GD_GeneralDynamics_zscore_60d", "HD_ret_20d", "DE_Deere_ret_5d", "EWM_Malaysia_zscore_60d", "IWM_SmallCap_vol_20d", "AXP_Amex_ret_20d"], "is_new": true}, {"model_id": "new_h5_CALM_RandomForest_N15_t2", "algo": "RandomForest", "regime": "CALM", "horizon": 5, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "VRP_ma5", "EWM_Malaysia_ret_1d", "vix_acceleration_1d", "XOM_ret_1d", "MS_MorganStanley_ret_1d", "AMZN_ret_5d", "MS_MorganStanley_ret_5d", "DIS_vol_20d", "NWL_Newell_ret_20d", "EWC_Canada_zscore_60d", "TM_Telephone_vol_20d", "EWM_Malaysia_vol_20d", "XLY_Disc_vol_20d"], "is_new": true}, {"model_id": "new_h5_CALM_RandomForest_N15_t3", "algo": "RandomForest", "regime": "CALM", "horizon": 5, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "SJM_JM_Smucker_ret_5d", "spx_momentum_3d", "PPL_PPL_ret_1d", "HD_ret_1d", "LMT_LockheedMartin_vol_20d", "EXC_Exelon_ret_1d", "LOW_Lowes_ret_5d", "NVDA_vol_20d", "AMD_ret_5d", "EWA_Australia_zscore_60d", "spx_vol_5d", "DAX_Germany_vol_20d", "hmm_p_stress"], "is_new": true}, {"model_id": "new_h5_CALM_RandomForest_N15_t4", "algo": "RandomForest", "regime": "CALM", "horizon": 5, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "US7Y_Rate_ret_20d", "EQR_Equity_ret_1d", "QQQ_vol_20d", "VVIX_ret_20d", "IBEX_Spain_ret_20d", "SPY_zscore_60d", "Core_CPI_zscore_60d", "gjr_condvar_h1", "EQIX_Equinix_ret_5d", "XLF_Fin_vol_20d", "heston_var_ev_h5", "NFCI_ret_5d", "HUM_Humana_ret_5d"], "is_new": true}, {"model_id": "new_h5_CALM_RandomForest_N15_t5", "algo": "RandomForest", "regime": "CALM", "horizon": 5, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "SCHW_Schwab_ret_5d", "DAX_Germany_vol_20d", "US3M_Rate_vol_20d", "LLY_zscore_60d", "XLB_Materials_zscore_60d", "CCI_CrownCastle_vol_20d", "heston_ev_h3", "XOM_ret_1d", "VOD_Vodafone_zscore_60d", "FedFunds_zscore_60d", "CTAS_Cintas_vol_20d", "EWM_Malaysia_vol_20d", "M_Macys_vol_20d"], "is_new": true}, {"model_id": "new_h5_CALM_RandomForest_N15_t6", "algo": "RandomForest", "regime": "CALM", "horizon": 5, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EMR_Emerson_ret_20d", "US6M_Rate_ret_20d", "CPB_CampbellSoup_ret_20d", "Core_PCE_zscore_60d", "TM_Telephone_vol_20d", "SJM_JM_Smucker_ret_5d", "HD_ret_20d", "IBEX_Spain_ret_20d", "AXP_Amex_ret_20d", "Core_CPI_zscore_60d", "EXC_Exelon_ret_1d", "PAYX_Paychex_vol_20d", "3M_ret_5d"], "is_new": true}, {"model_id": "new_h5_CALM_RandomForest_N15_t7", "algo": "RandomForest", "regime": "CALM", "horizon": 5, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "VRP_ma5", "MSTR_Bitcoin3_ret_1d", "CTAS_Cintas_vol_20d", "PPL_PPL_ret_1d", "hmm_p_stress", "DHR_vol_20d", "LMT_LockheedMartin_vol_20d", "SO_SouthernCo_ret_5d", "TXN_vol_20d", "DE_Deere_vol_20d", "spx_vol_5d", "SPY_zscore_60d", "EXC_Exelon_zscore_60d"], "is_new": true}, {"model_id": "new_h5_CALM_RandomForest_N20_t0", "algo": "RandomForest", "regime": "CALM", "horizon": 5, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "Core_PCE_zscore_60d", "heston_var_ev_h3", "SLB_Schlumberger_ret_1d", "AORD_AUS_zscore_60d", "FedFunds_zscore_60d", "LOW_Lowes_ret_5d", "IWM_SmallCap_vol_20d", "XLK_Tech_zscore_60d", "CMCSA_ret_1d", "EOG_EOGResources_vol_20d", "TM_Telephone_vol_20d", "XLF_Fin_vol_20d", "US3M_Rate_vol_20d", "MRK_Merck_zscore_60d", "Core_CPI_zscore_60d", "EMR_Emerson_ret_20d", "CPB_CampbellSoup_zscore_60d", "heston_ev_h3"], "is_new": true}, {"model_id": "new_h5_CALM_RandomForest_N20_t1", "algo": "RandomForest", "regime": "CALM", "horizon": 5, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "PAYX_Paychex_ret_20d", "MRK_Merck_zscore_60d", "NWL_Newell_ret_20d", "EWM_Malaysia_vol_20d", "MS_MorganStanley_ret_5d", "JNJ_ret_1d", "DAX_Germany_vol_20d", "SJM_JM_Smucker_ret_5d", "EWL_Switzerland_vol_20d", "PPL_PPL_ret_1d", "AMZN_ret_5d", "HangSeng_HK_ret_5d", "PFE_ret_1d", "AVB_AvalonBay_zscore_60d", "EMR_Emerson_ret_20d", "EWA_Australia_zscore_60d", "PCAR_PaccarInc_ret_5d", "spx_abs_ret_max_5d"], "is_new": true}, {"model_id": "new_h5_CALM_RandomForest_N20_t2", "algo": "RandomForest", "regime": "CALM", "horizon": 5, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "MSTR_Bitcoin3_ret_1d", "SLB_Schlumberger_ret_5d", "ES_Evergy_ret_1d", "CTAS_Cintas_vol_20d", "DHR_ret_1d", "3M_vol_20d", "EWL_Switzerland_zscore_60d", "MO_AltriaMG_ret_1d", "SO_SouthernCo_ret_5d", "Core_PCE_zscore_60d", "EWQ_France_zscore_60d", "XLV_Health_zscore_60d", "gjr_condvar_h1", "spx_momentum_3d", "PAYX_Paychex_ret_20d", "XLY_Disc_vol_20d", "spx_vol_5d", "IYM_BasicMaterials_ret_20d"], "is_new": true}, {"model_id": "new_h5_CALM_RandomForest_N20_t3", "algo": "RandomForest", "regime": "CALM", "horizon": 5, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "MRK_Merck_zscore_60d", "DOW_Price_zscore_60d", "FedFunds_zscore_60d", "ASX_Australia_ret_5d", "LUV_SouthwestAir_ret_5d", "LOW_Lowes_ret_5d", "SLB_Schlumberger_ret_1d", "US3M_Rate_vol_20d", "PLD_Prologis_ret_5d", "EOG_EOGResources_vol_20d", "EWM_Malaysia_zscore_60d", "VVIX_ret_20d", "BTI_BritishAmerican_ret_20d", "US3M_Rate_zscore_60d", "vix_acceleration_1d", "XOM_ret_1d", "SO_SouthernCo_ret_5d", "SPY_zscore_60d"], "is_new": true}, {"model_id": "new_h5_CALM_RandomForest_N20_t4", "algo": "RandomForest", "regime": "CALM", "horizon": 5, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "WTI_Oil_FRED_zscore_60d", "EFFR_ret_1d", "XLK_Tech_zscore_60d", "ORCL_zscore_60d", "SBUX_ret_5d", "ASX_Australia_ret_5d", "EQR_Equity_ret_1d", "LUV_SouthwestAir_ret_5d", "gjr_condvar_h1", "US7Y_Rate_ret_20d", "US1Y_Rate_ret_20d", "HD_ret_20d", "LMT_LockheedMartin_vol_20d", "US30Y_Rate_ret_20d", "INTC_ret_5d", "INTC_ret_1d", "DE_Deere_vol_20d", "EFFR_vol_20d"], "is_new": true}, {"model_id": "new_h5_CALM_RandomForest_N20_t5", "algo": "RandomForest", "regime": "CALM", "horizon": 5, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EQIX_Equinix_ret_5d", "US5Y_Rate_ret_5d", "CTAS_Cintas_vol_20d", "ES_Evergy_ret_1d", "WTI_Oil_FRED_zscore_60d", "SLB_Schlumberger_ret_1d", "spx_momentum_3d", "US3M_Rate_vol_20d", "M_Macys_vol_20d", "PAYX_Paychex_vol_20d", "EWQ_France_zscore_60d", "EWY_Korea_ret_20d", "hmm_p_stress", "LUV_SouthwestAir_ret_5d", "EWQ_France_ret_20d", "PCAR_PaccarInc_ret_5d", "AORD_AUS_zscore_60d", "gjr_condvar_h1"], "is_new": true}, {"model_id": "new_h5_CALM_RandomForest_N20_t6", "algo": "RandomForest", "regime": "CALM", "horizon": 5, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "BA_ret_1d", "3M_ret_5d", "XOM_ret_20d", "Brent_Oil_FRED_ret_5d", "ORCL_zscore_60d", "CI_Cigna_vol_20d", "XLF_Fin_vol_20d", "LMT_LockheedMartin_vol_20d", "LOW_Lowes_ret_20d", "Retail_Sales_zscore_60d", "Nikkei_Japan_zscore_60d", "EWA_Australia_ret_1d", "DE_Deere_ret_5d", "VVIX_ret_20d", "MRK_Merck_zscore_60d", "hmm_p_stress", "Core_PCE_zscore_60d", "MS_MorganStanley_ret_5d"], "is_new": true}, {"model_id": "new_h5_CALM_RandomForest_N20_t7", "algo": "RandomForest", "regime": "CALM", "horizon": 5, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "NWL_Newell_ret_20d", "heston_ev_h3", "QQQ_vol_20d", "GE_ret_1d", "US1Y_Rate_ret_5d", "MSTR_Bitcoin3_ret_1d", "EWG_Germany_vol_20d", "SJM_JM_Smucker_ret_1d", "HD_ret_20d", "SBUX_zscore_60d", "BA_ret_1d", "EWQ_France_ret_20d", "heston_var_ev_h5", "AMGN_Amgen_ret_1d", "EWJ_Japan_vol_20d", "NFCI_ret_5d", "MS_MorganStanley_ret_5d", "EWS_Singapore_ret_5d"], "is_new": true}, {"model_id": "new_h5_CALM_RandomForest_N25_t0", "algo": "RandomForest", "regime": "CALM", "horizon": 5, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "MSTR_Bitcoin3_ret_5d", "XOM_ret_20d", "SLB_Schlumberger_ret_5d", "PCAR_PaccarInc_ret_5d", "Core_CPI_zscore_60d", "spx_momentum_3d", "IYR_US_REIT2_zscore_60d", "SJM_JM_Smucker_ret_5d", "EXC_Exelon_zscore_60d", "AMT_AmericanTower_ret_1d", "heston_var_ev_h7", "EWM_Malaysia_ret_1d", "BTI_BritishAmerican_ret_5d", "TGT_Target_zscore_60d", "US1Y_Rate_ret_20d", "NEE_NextEra_ret_20d", "IYM_BasicMaterials_ret_20d", "TXN_vol_20d", "IWM_SmallCap_vol_20d", "EQR_Equity_ret_1d", "ES_Evergy_ret_1d", "SLB_Schlumberger_ret_1d", "SO_SouthernCo_ret_5d"], "is_new": true}, {"model_id": "new_h5_CALM_RandomForest_N25_t1", "algo": "RandomForest", "regime": "CALM", "horizon": 5, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "PAYX_Paychex_vol_20d", "EFFR_vol_20d", "PG_ret_20d", "GE_ret_1d", "NOC_Northrop_ret_20d", "BDX_Becton_Dickinson_ret_20d", "AMD_ret_1d", "EWS_Singapore_ret_5d", "HUM_Humana_ret_5d", "CMCSA_ret_1d", "3M_ret_5d", "MRK_Merck_zscore_60d", "US30Y_Rate_ret_20d", "US1Y_Rate_ret_5d", "EOG_EOGResources_vol_20d", "TED_Spread_zscore_60d", "PAYX_Paychex_zscore_60d", "AORD_AUS_zscore_60d", "EWQ_France_ret_20d", "XOM_ret_20d", "CLX_Clorox_vol_20d", "Retail_Sales_zscore_60d", "VRP_ma5"], "is_new": true}, {"model_id": "new_h5_CALM_RandomForest_N25_t2", "algo": "RandomForest", "regime": "CALM", "horizon": 5, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "Industrial_Production_zscore_60d", "AMD_ret_5d", "AMT_AmericanTower_ret_1d", "XLB_Materials_zscore_60d", "Retail_Sales_zscore_60d", "AXP_Amex_vol_20d", "Nikkei_Japan_vol_20d", "GD_GeneralDynamics_zscore_60d", "EXC_Exelon_zscore_60d", "hmm_p_stress", "QQQ_vol_20d", "Brent_Oil_FRED_ret_20d", "EOG_EOGResources_vol_20d", "ITT_ITTInc_ret_5d", "EWM_Malaysia_vol_20d", "HD_ret_5d", "NVDA_vol_20d", "CPB_CampbellSoup_zscore_60d", "CPB_CampbellSoup_ret_20d", "IWM_SmallCap_vol_20d", "IYM_BasicMaterials_ret_20d", "IYR_US_REIT2_zscore_60d", "EQR_Equity_ret_1d"], "is_new": true}, {"model_id": "new_h5_CALM_RandomForest_N25_t3", "algo": "RandomForest", "regime": "CALM", "horizon": 5, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "US30Y_Rate_ret_20d", "ES_Evergy_ret_1d", "BA_ret_1d", "Core_CPI_zscore_60d", "LOW_Lowes_ret_5d", "US1Y_Rate_ret_20d", "BTI_BritishAmerican_ret_20d", "spx_momentum_3d", "EWM_Malaysia_vol_20d", "AORD_AUS_zscore_60d", "SO_SouthernCo_ret_5d", "heston_var_ev_h3", "VVIX_ret_20d", "LMT_LockheedMartin_ret_1d", "EWA_Australia_ret_1d", "CTAS_Cintas_vol_20d", "Brent_Oil_FRED_ret_5d", "ORCL_vol_20d", "ITT_ITTInc_ret_5d", "DAX_Germany_vol_20d", "LMT_LockheedMartin_vol_20d", "DAX_Germany_zscore_60d", "DIS_vol_20d"], "is_new": true}, {"model_id": "new_h5_CALM_RandomForest_N25_t4", "algo": "RandomForest", "regime": "CALM", "horizon": 5, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "T_ret_1d", "US3M_Rate_vol_20d", "NWL_Newell_ret_20d", "AORD_AUS_zscore_60d", "Retail_Sales_zscore_60d", "Brent_Oil_FRED_ret_5d", "EWL_Switzerland_vol_20d", "SBUX_vol_20d", "MSTR_Bitcoin3_ret_5d", "heston_var_ev_h3", "PLD_Prologis_ret_5d", "AMD_ret_5d", "US30Y_Rate_ret_20d", "BLK_BlackRock_zscore_60d", "heston_ev_h3", "AMD_ret_1d", "Nikkei_Japan_vol_20d", "BA_ret_1d", "JNJ_ret_1d", "XLY_Disc_vol_20d", "NVDA_vol_20d", "HD_zscore_60d", "spx_momentum_3d"], "is_new": true}, {"model_id": "new_h5_CALM_RandomForest_N25_t5", "algo": "RandomForest", "regime": "CALM", "horizon": 5, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EQIX_Equinix_ret_5d", "US1Y_Rate_ret_20d", "GILD_Gilead_ret_20d", "AMT_AmericanTower_ret_1d", "CPB_CampbellSoup_ret_20d", "HUM_Humana_ret_5d", "HD_ret_1d", "AVB_AvalonBay_zscore_60d", "PCAR_PaccarInc_ret_5d", "XLY_Disc_vol_20d", "spx_momentum_3d", "GE_ret_1d", "EWG_Germany_ret_20d", "SPY_zscore_60d", "US3M_Rate_vol_20d", "EWA_Australia_zscore_60d", "ITT_ITTInc_ret_5d", "EWM_Malaysia_ret_1d", "CPB_CampbellSoup_ret_5d", "heston_var_ev_h3", "EWA_Australia_ret_1d", "DIS_vol_20d", "HD_zscore_60d"], "is_new": true}, {"model_id": "new_h5_CALM_RandomForest_N25_t6", "algo": "RandomForest", "regime": "CALM", "horizon": 5, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EFFR_vol_20d", "EFFR_ret_1d", "HD_zscore_60d", "hmm_p_stress", "CCI_CrownCastle_vol_20d", "BTI_BritishAmerican_ret_5d", "EWY_Korea_ret_20d", "EXC_Exelon_zscore_60d", "spx_abs_ret_max_5d", "HUM_Humana_ret_5d", "US1Y_Rate_ret_20d", "CPB_CampbellSoup_vol_20d", "DHR_ret_1d", "NOC_Northrop_ret_20d", "T_ret_1d", "US3M_Rate_vol_20d", "DE_Deere_vol_20d", "MSTR_Bitcoin3_ret_5d", "MO_AltriaMG_ret_1d", "TM_Telephone_vol_20d", "EWL_Switzerland_zscore_60d", "IWM_SmallCap_vol_20d", "EOG_EOGResources_vol_20d"], "is_new": true}, {"model_id": "new_h5_CALM_RandomForest_N25_t7", "algo": "RandomForest", "regime": "CALM", "horizon": 5, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "SJM_JM_Smucker_ret_5d", "TXN_vol_20d", "ITT_ITTInc_ret_5d", "NEE_NextEra_ret_20d", "AMZN_ret_5d", "spx_momentum_3d", "XLK_Tech_zscore_60d", "GD_GeneralDynamics_zscore_60d", "IWM_SmallCap_vol_20d", "Michigan_Sentiment_ret_20d", "SLB_Schlumberger_ret_5d", "GILD_Gilead_ret_20d", "ORCL_zscore_60d", "BA_ret_1d", "XLF_Fin_vol_20d", "EWL_Switzerland_vol_20d", "CCI_CrownCastle_vol_20d", "MRK_Merck_zscore_60d", "SBUX_zscore_60d", "US3M_Rate_zscore_60d", "VOD_Vodafone_zscore_60d", "Nikkei_Japan_vol_20d", "IYM_BasicMaterials_ret_20d"], "is_new": true}, {"model_id": "new_h5_CALM_RandomForest_N30_t0", "algo": "RandomForest", "regime": "CALM", "horizon": 5, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "VVIX_ret_20d", "EWY_Korea_zscore_60d", "EXC_Exelon_ret_1d", "heston_var_ev_h7", "XLF_Fin_vol_20d", "spx_vol_5d", "XLV_Health_zscore_60d", "EWL_Switzerland_vol_20d", "3M_vol_20d", "PG_ret_20d", "US1Y_Rate_ret_20d", "IBEX_Spain_ret_20d", "IYM_BasicMaterials_ret_20d", "PCAR_PaccarInc_ret_5d", "SJM_JM_Smucker_ret_5d", "FedFunds_zscore_60d", "vix_acceleration_1d", "vix_mean_abs_ret_5d", "BLK_BlackRock_zscore_60d", "EOG_EOGResources_ret_5d", "EWM_Malaysia_zscore_60d", "BDX_Becton_Dickinson_ret_20d", "Retail_Sales_zscore_60d", "QQQ_vol_20d", "Core_CPI_zscore_60d", "XOM_ret_20d", "MSTR_Bitcoin3_ret_5d", "XOM_ret_1d"], "is_new": true}, {"model_id": "new_h5_CALM_RandomForest_N30_t1", "algo": "RandomForest", "regime": "CALM", "horizon": 5, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "AMD_ret_5d", "Brent_Oil_FRED_ret_5d", "vix_acceleration_1d", "SLB_Schlumberger_ret_1d", "EWY_Korea_ret_20d", "PAYX_Paychex_ret_20d", "NEE_NextEra_ret_20d", "XLV_Health_zscore_60d", "BTI_BritishAmerican_ret_5d", "ASX_Australia_ret_5d", "SLB_Schlumberger_ret_5d", "Brent_Oil_FRED_ret_20d", "TXN_vol_20d", "WTI_Oil_FRED_zscore_60d", "TM_Telephone_vol_20d", "CPB_CampbellSoup_ret_20d", "EWL_Switzerland_zscore_60d", "QQQ_vol_20d", "TED_Spread_zscore_60d", "BDX_Becton_Dickinson_ret_20d", "CTAS_Cintas_vol_20d", "VRP_ma5", "AXP_Amex_ret_20d", "HD_ret_20d", "PFE_ret_1d", "3M_ret_5d", "gjr_condvar_h1", "GD_GeneralDynamics_zscore_60d"], "is_new": true}, {"model_id": "new_h5_CALM_RandomForest_N30_t2", "algo": "RandomForest", "regime": "CALM", "horizon": 5, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWM_Malaysia_vol_20d", "DE_Deere_vol_20d", "LMT_LockheedMartin_ret_1d", "Michigan_Sentiment_ret_20d", "PPL_PPL_ret_1d", "vix_acceleration_1d", "CLX_Clorox_vol_20d", "BTI_BritishAmerican_ret_20d", "US3M_Rate_vol_20d", "IYM_BasicMaterials_ret_20d", "Brent_Oil_FRED_ret_5d", "heston_var_ev_h7", "3M_vol_20d", "US3M_Rate_zscore_60d", "GE_ret_1d", "LOW_Lowes_ret_5d", "DE_Deere_ret_5d", "M_Macys_vol_20d", "LMT_LockheedMartin_vol_20d", "TED_Spread_vol_20d", "XLF_Fin_vol_20d", "Brent_Oil_FRED_ret_20d", "NEE_NextEra_ret_20d", "JNJ_ret_1d", "LOW_Lowes_ret_20d", "XOM_ret_20d", "HangSeng_HK_ret_1d", "ORCL_vol_20d"], "is_new": true}, {"model_id": "new_h5_CALM_RandomForest_N30_t3", "algo": "RandomForest", "regime": "CALM", "horizon": 5, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "TGT_Target_zscore_60d", "SO_SouthernCo_ret_5d", "TM_Telephone_vol_20d", "T10Y2Y_Spread_ret_5d", "spx_abs_ret_max_5d", "PPL_PPL_ret_1d", "heston_var_ev_h5", "NVDA_vol_20d", "HangSeng_HK_ret_1d", "CPB_CampbellSoup_zscore_60d", "HangSeng_HK_vol_20d", "EWQ_France_zscore_60d", "PFE_ret_1d", "PAYX_Paychex_ret_20d", "ENB_EnbridgeInc_ret_1d", "EFFR_vol_20d", "EQIX_Equinix_ret_5d", "SLB_Schlumberger_ret_1d", "MO_AltriaMG_ret_1d", "IWM_SmallCap_vol_20d", "Nikkei_Japan_vol_20d", "US6M_Rate_ret_20d", "EWA_Australia_ret_1d", "CCI_CrownCastle_vol_20d", "HD_ret_1d", "US3M_Rate_vol_20d", "EWM_Malaysia_zscore_60d", "LMT_LockheedMartin_vol_20d"], "is_new": true}, {"model_id": "new_h5_CALM_RandomForest_N30_t4", "algo": "RandomForest", "regime": "CALM", "horizon": 5, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWG_Germany_ret_20d", "DHR_vol_20d", "PG_ret_20d", "NOC_Northrop_ret_20d", "Brent_Oil_FRED_ret_20d", "VRP_ma5", "XLF_Fin_vol_20d", "AVB_AvalonBay_zscore_60d", "EWQ_France_ret_20d", "PAYX_Paychex_vol_20d", "PPL_PPL_ret_1d", "PAYX_Paychex_zscore_60d", "EWY_Korea_ret_20d", "CLX_Clorox_vol_20d", "CPB_CampbellSoup_ret_20d", "AMD_ret_1d", "EOG_EOGResources_ret_5d", "INTC_ret_5d", "EWA_Australia_zscore_60d", "PCAR_PaccarInc_ret_5d", "XLB_Materials_zscore_60d", "US6M_Rate_ret_20d", "EWL_Switzerland_zscore_60d", "EWH_HongKong_ret_5d", "IYM_BasicMaterials_ret_20d", "TXN_vol_20d", "INTC_ret_1d", "Core_CPI_zscore_60d"], "is_new": true}, {"model_id": "new_h5_CALM_RandomForest_N30_t5", "algo": "RandomForest", "regime": "CALM", "horizon": 5, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "US5Y_Rate_ret_5d", "GILD_Gilead_ret_20d", "AVB_AvalonBay_zscore_60d", "AMT_AmericanTower_ret_1d", "SCHW_Schwab_ret_5d", "FedFunds_zscore_60d", "Nikkei_Japan_zscore_60d", "QQQ_vol_20d", "DE_Deere_vol_20d", "EWM_Malaysia_vol_20d", "VOD_Vodafone_zscore_60d", "PFE_ret_1d", "XOM_ret_1d", "T10Y2Y_Spread_ret_5d", "TM_Telephone_ret_1d", "CCI_CrownCastle_vol_20d", "CPB_CampbellSoup_vol_20d", "TED_Spread_vol_20d", "SLB_Schlumberger_ret_1d", "DHR_vol_20d", "CPB_CampbellSoup_ret_20d", "AXP_Amex_vol_20d", "ORCL_zscore_60d", "MSTR_Bitcoin3_ret_1d", "ASX_Australia_ret_5d", "ITT_ITTInc_ret_5d", "LUV_SouthwestAir_ret_5d", "INTC_ret_5d"], "is_new": true}, {"model_id": "new_h5_CALM_RandomForest_N30_t6", "algo": "RandomForest", "regime": "CALM", "horizon": 5, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "DHR_vol_20d", "MO_AltriaMG_ret_1d", "EWM_Malaysia_ret_1d", "ORCL_zscore_60d", "PLD_Prologis_ret_5d", "NEE_NextEra_ret_20d", "VRP_ma5", "hmm_p_stress", "Retail_Sales_zscore_60d", "EWY_Korea_ret_20d", "DE_Deere_ret_5d", "MRK_Merck_zscore_60d", "GE_ret_1d", "VOD_Vodafone_zscore_60d", "MS_MorganStanley_ret_5d", "MSTR_Bitcoin3_ret_20d", "AORD_AUS_zscore_60d", "SBUX_vol_20d", "3M_vol_20d", "EQIX_Equinix_ret_5d", "T_ret_1d", "PPL_PPL_ret_1d", "EWJ_Japan_vol_20d", "SCHW_Schwab_ret_5d", "DAX_Germany_zscore_60d", "EWH_HongKong_ret_5d", "heston_var_ev_h7", "XLF_Fin_vol_20d"], "is_new": true}, {"model_id": "new_h5_CALM_RandomForest_N30_t7", "algo": "RandomForest", "regime": "CALM", "horizon": 5, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "Core_CPI_zscore_60d", "Michigan_Sentiment_ret_20d", "ORCL_vol_20d", "EQR_Equity_ret_1d", "XLB_Materials_zscore_60d", "MSTR_Bitcoin3_ret_5d", "hmm_p_stress", "T10Y2Y_Spread_ret_5d", "gjr_condvar_h1", "spx_momentum_3d", "Retail_Sales_zscore_60d", "EWM_Malaysia_vol_20d", "FedFunds_zscore_60d", "EOG_EOGResources_ret_5d", "TED_Spread_vol_20d", "US3M_Rate_vol_20d", "BLK_BlackRock_zscore_60d", "XOM_ret_1d", "XLY_Disc_vol_20d", "MS_MorganStanley_zscore_60d", "TM_Telephone_vol_20d", "IWM_SmallCap_vol_20d", "HD_zscore_60d", "US7Y_Rate_ret_20d", "MS_MorganStanley_ret_5d", "3M_ret_5d", "vix_acceleration_1d", "SLB_Schlumberger_ret_1d"], "is_new": true}, {"model_id": "new_h5_CALM_LogisticRegression_N5_t0", "algo": "LogisticRegression", "regime": "CALM", "horizon": 5, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "Brent_Oil_FRED_ret_20d", "XOM_ret_1d", "Brent_Oil_FRED_ret_5d"], "is_new": true}, {"model_id": "new_h5_CALM_LogisticRegression_N5_t1", "algo": "LogisticRegression", "regime": "CALM", "horizon": 5, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "HD_ret_1d", "HangSeng_HK_ret_1d", "DAX_Germany_zscore_60d"], "is_new": true}, {"model_id": "new_h5_CALM_LogisticRegression_N5_t2", "algo": "LogisticRegression", "regime": "CALM", "horizon": 5, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "HangSeng_HK_ret_1d", "TGT_Target_zscore_60d", "LLY_zscore_60d"], "is_new": true}, {"model_id": "new_h5_CALM_LogisticRegression_N5_t3", "algo": "LogisticRegression", "regime": "CALM", "horizon": 5, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "HD_ret_20d", "IBEX_Spain_ret_20d", "NOC_Northrop_ret_20d"], "is_new": true}, {"model_id": "new_h5_CALM_LogisticRegression_N5_t4", "algo": "LogisticRegression", "regime": "CALM", "horizon": 5, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "hmm_p_stress", "HD_zscore_60d", "CPB_CampbellSoup_ret_20d"], "is_new": true}, {"model_id": "new_h5_CALM_LogisticRegression_N5_t5", "algo": "LogisticRegression", "regime": "CALM", "horizon": 5, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "spx_momentum_3d", "MSTR_Bitcoin3_ret_5d", "HD_ret_5d"], "is_new": true}, {"model_id": "new_h5_CALM_LogisticRegression_N5_t6", "algo": "LogisticRegression", "regime": "CALM", "horizon": 5, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "BLK_BlackRock_zscore_60d", "EWL_Switzerland_zscore_60d", "ORCL_zscore_60d"], "is_new": true}, {"model_id": "new_h5_CALM_LogisticRegression_N5_t7", "algo": "LogisticRegression", "regime": "CALM", "horizon": 5, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "GD_GeneralDynamics_zscore_60d", "DAX_Germany_zscore_60d", "Core_CPI_zscore_60d"], "is_new": true}, {"model_id": "new_h5_CALM_LogisticRegression_N8_t0", "algo": "LogisticRegression", "regime": "CALM", "horizon": 5, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "AMT_AmericanTower_ret_1d", "MRK_Merck_zscore_60d", "SJM_JM_Smucker_ret_5d", "AMZN_ret_5d", "TM_Telephone_ret_1d", "IWM_SmallCap_vol_20d"], "is_new": true}, {"model_id": "new_h5_CALM_LogisticRegression_N8_t1", "algo": "LogisticRegression", "regime": "CALM", "horizon": 5, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "XOM_ret_20d", "SJM_JM_Smucker_ret_1d", "BA_ret_1d", "SJM_JM_Smucker_ret_5d", "SLB_Schlumberger_ret_1d", "AMD_ret_1d"], "is_new": true}, {"model_id": "new_h5_CALM_LogisticRegression_N8_t2", "algo": "LogisticRegression", "regime": "CALM", "horizon": 5, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "BLK_BlackRock_zscore_60d", "Retail_Sales_zscore_60d", "IYM_BasicMaterials_ret_20d", "EWQ_France_ret_20d", "AMGN_Amgen_ret_1d", "US30Y_Rate_ret_20d"], "is_new": true}, {"model_id": "new_h5_CALM_LogisticRegression_N8_t3", "algo": "LogisticRegression", "regime": "CALM", "horizon": 5, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "DOW_Price_zscore_60d", "MSTR_Bitcoin3_ret_20d", "WTI_Oil_FRED_zscore_60d", "NWL_Newell_ret_20d", "MS_MorganStanley_ret_5d", "PAYX_Paychex_ret_20d"], "is_new": true}, {"model_id": "new_h5_CALM_LogisticRegression_N8_t4", "algo": "LogisticRegression", "regime": "CALM", "horizon": 5, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "BLK_BlackRock_zscore_60d", "CLX_Clorox_vol_20d", "IWM_SmallCap_vol_20d", "ASX_Australia_vol_20d", "AXP_Amex_ret_20d", "TED_Spread_zscore_60d"], "is_new": true}, {"model_id": "new_h5_CALM_LogisticRegression_N8_t5", "algo": "LogisticRegression", "regime": "CALM", "horizon": 5, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "HD_zscore_60d", "MS_MorganStanley_ret_1d", "LMT_LockheedMartin_ret_1d", "BTI_BritishAmerican_ret_5d", "ORCL_vol_20d", "SJM_JM_Smucker_ret_1d"], "is_new": true}, {"model_id": "new_h5_CALM_LogisticRegression_N8_t6", "algo": "LogisticRegression", "regime": "CALM", "horizon": 5, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "ORCL_zscore_60d", "BTI_BritishAmerican_ret_20d", "XLB_Materials_zscore_60d", "INTC_ret_5d", "DHR_vol_20d", "Michigan_Sentiment_ret_20d"], "is_new": true}, {"model_id": "new_h5_CALM_LogisticRegression_N8_t7", "algo": "LogisticRegression", "regime": "CALM", "horizon": 5, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "PPL_PPL_ret_1d", "BLK_BlackRock_zscore_60d", "vix_acceleration_1d", "US7Y_Rate_ret_20d", "Nikkei_Japan_zscore_60d", "SLB_Schlumberger_ret_1d"], "is_new": true}, {"model_id": "new_h5_CALM_LogisticRegression_N10_t0", "algo": "LogisticRegression", "regime": "CALM", "horizon": 5, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "BTI_BritishAmerican_ret_5d", "ORCL_vol_20d", "EXC_Exelon_ret_1d", "TM_Telephone_vol_20d", "hmm_p_stress", "INTC_ret_1d", "EXC_Exelon_zscore_60d", "PG_ret_20d"], "is_new": true}, {"model_id": "new_h5_CALM_LogisticRegression_N10_t1", "algo": "LogisticRegression", "regime": "CALM", "horizon": 5, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "vix_acceleration_1d", "IWM_SmallCap_vol_20d", "US1Y_Rate_ret_20d", "CMCSA_ret_1d", "US7Y_Rate_ret_20d", "NEE_NextEra_ret_20d", "EWH_HongKong_ret_5d", "EWY_Korea_zscore_60d"], "is_new": true}, {"model_id": "new_h5_CALM_LogisticRegression_N10_t2", "algo": "LogisticRegression", "regime": "CALM", "horizon": 5, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "ASX_Australia_vol_20d", "HangSeng_HK_ret_5d", "EWQ_France_zscore_60d", "MSTR_Bitcoin3_ret_5d", "DHR_vol_20d", "MRK_Merck_zscore_60d", "heston_var_ev_h5", "MS_MorganStanley_ret_5d"], "is_new": true}, {"model_id": "new_h5_CALM_LogisticRegression_N10_t3", "algo": "LogisticRegression", "regime": "CALM", "horizon": 5, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "heston_var_ev_h5", "XLB_Materials_zscore_60d", "EWQ_France_zscore_60d", "3M_vol_20d", "JNJ_ret_1d", "US1Y_Rate_ret_5d", "CCI_CrownCastle_vol_20d", "TM_Telephone_ret_1d"], "is_new": true}, {"model_id": "new_h5_CALM_LogisticRegression_N10_t4", "algo": "LogisticRegression", "regime": "CALM", "horizon": 5, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "3M_ret_5d", "HD_ret_1d", "Nikkei_Japan_vol_20d", "MSTR_Bitcoin3_ret_1d", "CMCSA_ret_1d", "AORD_AUS_zscore_60d", "BTI_BritishAmerican_ret_20d", "ASX_Australia_ret_5d"], "is_new": true}, {"model_id": "new_h5_CALM_LogisticRegression_N10_t5", "algo": "LogisticRegression", "regime": "CALM", "horizon": 5, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "MSTR_Bitcoin3_ret_1d", "CMCSA_ret_1d", "TED_Spread_zscore_60d", "XLK_Tech_zscore_60d", "EWG_Germany_ret_20d", "DOW_Price_zscore_60d", "US3Y_Rate_ret_5d", "TM_Telephone_vol_20d"], "is_new": true}, {"model_id": "new_h5_CALM_LogisticRegression_N10_t6", "algo": "LogisticRegression", "regime": "CALM", "horizon": 5, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "MSTR_Bitcoin3_ret_5d", "CTAS_Cintas_vol_20d", "HangSeng_HK_ret_1d", "LOW_Lowes_ret_20d", "AMGN_Amgen_ret_1d", "vix_mean_abs_ret_5d", "Brent_Oil_FRED_ret_5d", "US1Y_Rate_ret_20d"], "is_new": true}, {"model_id": "new_h5_CALM_LogisticRegression_N10_t7", "algo": "LogisticRegression", "regime": "CALM", "horizon": 5, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWM_Malaysia_vol_20d", "PG_ret_20d", "HangSeng_HK_ret_5d", "EWG_Germany_ret_20d", "MSTR_Bitcoin3_ret_5d", "heston_var_ev_h7", "Brent_Oil_FRED_ret_5d", "EQIX_Equinix_ret_5d"], "is_new": true}, {"model_id": "new_h5_CALM_LogisticRegression_N12_t0", "algo": "LogisticRegression", "regime": "CALM", "horizon": 5, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "spx_vol_5d", "spx_momentum_3d", "MS_MorganStanley_ret_1d", "CPB_CampbellSoup_ret_20d", "TED_Spread_vol_20d", "XLB_Materials_zscore_60d", "CCI_CrownCastle_vol_20d", "DE_Deere_ret_5d", "IBEX_Spain_ret_20d", "BTI_BritishAmerican_ret_20d"], "is_new": true}, {"model_id": "new_h5_CALM_LogisticRegression_N12_t1", "algo": "LogisticRegression", "regime": "CALM", "horizon": 5, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "CCI_CrownCastle_vol_20d", "spx_abs_ret_max_5d", "XOM_ret_20d", "XOM_ret_1d", "EWQ_France_zscore_60d", "EWJ_Japan_vol_20d", "TXN_vol_20d", "PCAR_PaccarInc_ret_5d", "EWG_Germany_vol_20d", "AMGN_Amgen_ret_1d"], "is_new": true}, {"model_id": "new_h5_CALM_LogisticRegression_N12_t2", "algo": "LogisticRegression", "regime": "CALM", "horizon": 5, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "hmm_p_stress", "EFFR_vol_20d", "SLB_Schlumberger_ret_1d", "VOD_Vodafone_zscore_60d", "PAYX_Paychex_ret_20d", "XLB_Materials_zscore_60d", "LOW_Lowes_ret_20d", "PG_ret_20d", "EWM_Malaysia_vol_20d", "PLD_Prologis_ret_5d"], "is_new": true}, {"model_id": "new_h5_CALM_LogisticRegression_N12_t3", "algo": "LogisticRegression", "regime": "CALM", "horizon": 5, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "gjr_condvar_h1", "HUM_Humana_ret_5d", "heston_var_ev_h5", "CPB_CampbellSoup_vol_20d", "AMT_AmericanTower_ret_1d", "T_ret_1d", "VRP_ma5", "HangSeng_HK_vol_20d", "MO_AltriaMG_ret_1d", "DHR_ret_1d"], "is_new": true}, {"model_id": "new_h5_CALM_LogisticRegression_N12_t4", "algo": "LogisticRegression", "regime": "CALM", "horizon": 5, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "XLF_Fin_vol_20d", "GE_ret_1d", "3M_vol_20d", "EWA_Australia_zscore_60d", "SCHW_Schwab_ret_5d", "MSTR_Bitcoin3_ret_5d", "FedFunds_zscore_60d", "US7Y_Rate_ret_20d", "HangSeng_HK_ret_1d", "CCI_CrownCastle_vol_20d"], "is_new": true}, {"model_id": "new_h5_CALM_LogisticRegression_N12_t5", "algo": "LogisticRegression", "regime": "CALM", "horizon": 5, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "INTC_ret_5d", "LOW_Lowes_ret_5d", "EFFR_ret_1d", "DE_Deere_vol_20d", "MRK_Merck_zscore_60d", "ITT_ITTInc_ret_5d", "HD_zscore_60d", "XOM_ret_1d", "EXC_Exelon_ret_1d", "DAX_Germany_zscore_60d"], "is_new": true}, {"model_id": "new_h5_CALM_LogisticRegression_N12_t6", "algo": "LogisticRegression", "regime": "CALM", "horizon": 5, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "CMCSA_ret_1d", "EWQ_France_ret_20d", "MS_MorganStanley_ret_5d", "EWG_Germany_ret_20d", "DIS_vol_20d", "LMT_LockheedMartin_ret_1d", "INTC_ret_1d", "TED_Spread_zscore_60d", "Nikkei_Japan_zscore_60d", "CPB_CampbellSoup_zscore_60d"], "is_new": true}, {"model_id": "new_h5_CALM_LogisticRegression_N12_t7", "algo": "LogisticRegression", "regime": "CALM", "horizon": 5, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "AMD_ret_1d", "LMT_LockheedMartin_ret_1d", "Nikkei_Japan_vol_20d", "CPB_CampbellSoup_ret_5d", "XOM_ret_20d", "EWM_Malaysia_vol_20d", "CPB_CampbellSoup_zscore_60d", "Brent_Oil_FRED_ret_5d", "DAX_Germany_vol_20d", "vix_mean_abs_ret_5d"], "is_new": true}, {"model_id": "new_h5_CALM_LogisticRegression_N15_t0", "algo": "LogisticRegression", "regime": "CALM", "horizon": 5, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWG_Germany_vol_20d", "NVDA_vol_20d", "PPL_PPL_ret_1d", "XLY_Disc_vol_20d", "BA_ret_1d", "SLB_Schlumberger_ret_5d", "EQR_Equity_ret_1d", "DAX_Germany_vol_20d", "EWJ_Japan_vol_20d", "MO_AltriaMG_ret_1d", "gjr_condvar_h1", "EQIX_Equinix_ret_5d", "DOW_Price_zscore_60d"], "is_new": true}, {"model_id": "new_h5_CALM_LogisticRegression_N15_t1", "algo": "LogisticRegression", "regime": "CALM", "horizon": 5, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "SPY_zscore_60d", "US7Y_Rate_ret_20d", "AORD_AUS_zscore_60d", "US3Y_Rate_ret_5d", "T_ret_1d", "EWL_Switzerland_vol_20d", "spx_vol_5d", "AXP_Amex_vol_20d", "SLB_Schlumberger_ret_1d", "SBUX_ret_5d", "US3M_Rate_zscore_60d", "EWJ_Japan_vol_20d", "EWL_Switzerland_zscore_60d"], "is_new": true}, {"model_id": "new_h5_CALM_LogisticRegression_N15_t2", "algo": "LogisticRegression", "regime": "CALM", "horizon": 5, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "SPY_zscore_60d", "hmm_p_stress", "XLB_Materials_zscore_60d", "CI_Cigna_vol_20d", "gjr_condvar_h1", "heston_var_ev_h7", "XLF_Fin_vol_20d", "US30Y_Rate_ret_20d", "NWL_Newell_ret_20d", "AMT_AmericanTower_ret_1d", "EMR_Emerson_ret_20d", "AMGN_Amgen_ret_1d", "SLB_Schlumberger_ret_5d"], "is_new": true}, {"model_id": "new_h5_CALM_LogisticRegression_N15_t3", "algo": "LogisticRegression", "regime": "CALM", "horizon": 5, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "GD_GeneralDynamics_zscore_60d", "EWQ_France_ret_20d", "EWG_Germany_vol_20d", "INTC_ret_5d", "heston_var_ev_h5", "SPY_zscore_60d", "NVDA_vol_20d", "EQIX_Equinix_ret_5d", "TM_Telephone_vol_20d", "EWY_Korea_ret_20d", "MO_AltriaMG_ret_1d", "SLB_Schlumberger_ret_1d", "US30Y_Rate_ret_20d"], "is_new": true}, {"model_id": "new_h5_CALM_LogisticRegression_N15_t4", "algo": "LogisticRegression", "regime": "CALM", "horizon": 5, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "spx_momentum_3d", "VRP_ma5", "ORCL_vol_20d", "SJM_JM_Smucker_ret_5d", "ASX_Australia_vol_20d", "PFE_ret_1d", "NEE_NextEra_ret_20d", "US1Y_Rate_ret_5d", "SCHW_Schwab_ret_5d", "US3M_Rate_zscore_60d", "BA_ret_1d", "EQIX_Equinix_ret_5d", "HD_ret_1d"], "is_new": true}, {"model_id": "new_h5_CALM_LogisticRegression_N15_t5", "algo": "LogisticRegression", "regime": "CALM", "horizon": 5, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EFFR_vol_20d", "EWG_Germany_ret_20d", "spx_abs_ret_max_5d", "LLY_zscore_60d", "US5Y_Rate_ret_5d", "TED_Spread_zscore_60d", "heston_ev_h3", "MSTR_Bitcoin3_ret_20d", "GE_ret_1d", "SO_SouthernCo_ret_5d", "AORD_AUS_zscore_60d", "US7Y_Rate_ret_20d", "ES_Evergy_ret_1d"], "is_new": true}, {"model_id": "new_h5_CALM_LogisticRegression_N15_t6", "algo": "LogisticRegression", "regime": "CALM", "horizon": 5, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "US5Y_Rate_ret_5d", "AVB_AvalonBay_zscore_60d", "SBUX_vol_20d", "MRK_Merck_zscore_60d", "EXC_Exelon_ret_1d", "DIS_vol_20d", "EFFR_ret_1d", "heston_var_ev_h3", "FedFunds_zscore_60d", "US1Y_Rate_ret_20d", "US7Y_Rate_ret_20d", "ASX_Australia_vol_20d", "CMCSA_ret_1d"], "is_new": true}, {"model_id": "new_h5_CALM_LogisticRegression_N15_t7", "algo": "LogisticRegression", "regime": "CALM", "horizon": 5, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "M_Macys_vol_20d", "EWA_Australia_ret_1d", "heston_var_ev_h7", "XLF_Fin_vol_20d", "CPB_CampbellSoup_zscore_60d", "IYR_US_REIT2_zscore_60d", "XOM_ret_1d", "MS_MorganStanley_ret_5d", "LOW_Lowes_ret_20d", "SLB_Schlumberger_ret_5d", "MSTR_Bitcoin3_ret_1d", "NOC_Northrop_ret_20d", "TED_Spread_vol_20d"], "is_new": true}, {"model_id": "new_h5_CALM_LogisticRegression_N20_t0", "algo": "LogisticRegression", "regime": "CALM", "horizon": 5, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWY_Korea_ret_20d", "GE_ret_1d", "TM_Telephone_ret_1d", "EWY_Korea_zscore_60d", "JNJ_ret_1d", "GILD_Gilead_ret_20d", "IWM_SmallCap_vol_20d", "EWQ_France_zscore_60d", "NVDA_vol_20d", "PCAR_PaccarInc_ret_5d", "US3M_Rate_zscore_60d", "EWA_Australia_ret_1d", "hmm_p_stress", "EOG_EOGResources_vol_20d", "NOC_Northrop_ret_20d", "EWL_Switzerland_vol_20d", "EXC_Exelon_zscore_60d", "LLY_zscore_60d"], "is_new": true}, {"model_id": "new_h5_CALM_LogisticRegression_N20_t1", "algo": "LogisticRegression", "regime": "CALM", "horizon": 5, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "Brent_Oil_FRED_ret_20d", "QQQ_vol_20d", "LOW_Lowes_ret_20d", "VRP_ma5", "MSTR_Bitcoin3_ret_1d", "NEE_NextEra_ret_20d", "CLX_Clorox_vol_20d", "HD_zscore_60d", "HangSeng_HK_vol_20d", "spx_momentum_3d", "NWL_Newell_ret_20d", "MS_MorganStanley_ret_1d", "MRK_Merck_zscore_60d", "SPY_zscore_60d", "vix_acceleration_1d", "GD_GeneralDynamics_zscore_60d", "US7Y_Rate_ret_20d", "BLK_BlackRock_zscore_60d"], "is_new": true}, {"model_id": "new_h5_CALM_LogisticRegression_N20_t2", "algo": "LogisticRegression", "regime": "CALM", "horizon": 5, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "AXP_Amex_vol_20d", "SBUX_ret_5d", "AXP_Amex_ret_20d", "MSTR_Bitcoin3_ret_5d", "Core_CPI_zscore_60d", "CPB_CampbellSoup_ret_20d", "EXC_Exelon_ret_1d", "AMD_ret_5d", "EQIX_Equinix_ret_5d", "ENB_EnbridgeInc_ret_1d", "spx_momentum_3d", "EWJ_Japan_vol_20d", "heston_var_ev_h3", "Industrial_Production_zscore_60d", "VOD_Vodafone_zscore_60d", "NOC_Northrop_ret_20d", "AMD_ret_1d", "MSTR_Bitcoin3_ret_1d"], "is_new": true}, {"model_id": "new_h5_CALM_LogisticRegression_N20_t3", "algo": "LogisticRegression", "regime": "CALM", "horizon": 5, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EQIX_Equinix_ret_5d", "VOD_Vodafone_zscore_60d", "LOW_Lowes_ret_5d", "AMT_AmericanTower_ret_1d", "VVIX_ret_20d", "vix_mean_abs_ret_5d", "HangSeng_HK_ret_5d", "HD_ret_5d", "EWJ_Japan_vol_20d", "ORCL_vol_20d", "AMGN_Amgen_ret_1d", "GE_ret_1d", "TGT_Target_zscore_60d", "SO_SouthernCo_ret_5d", "spx_momentum_3d", "MS_MorganStanley_zscore_60d", "BTI_BritishAmerican_ret_20d", "CPB_CampbellSoup_zscore_60d"], "is_new": true}, {"model_id": "new_h5_CALM_LogisticRegression_N20_t4", "algo": "LogisticRegression", "regime": "CALM", "horizon": 5, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "INTC_ret_1d", "ORCL_zscore_60d", "Michigan_Sentiment_ret_20d", "T10Y2Y_Spread_ret_5d", "DE_Deere_ret_5d", "Nikkei_Japan_zscore_60d", "hmm_p_stress", "EWJ_Japan_vol_20d", "ORCL_vol_20d", "WTI_Oil_FRED_zscore_60d", "AMD_ret_5d", "heston_ev_h3", "ASX_Australia_vol_20d", "SLB_Schlumberger_ret_1d", "US6M_Rate_ret_20d", "spx_momentum_3d", "QQQ_vol_20d", "EWM_Malaysia_zscore_60d"], "is_new": true}, {"model_id": "new_h5_CALM_LogisticRegression_N20_t5", "algo": "LogisticRegression", "regime": "CALM", "horizon": 5, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "LMT_LockheedMartin_vol_20d", "XLV_Health_zscore_60d", "ASX_Australia_ret_5d", "PLD_Prologis_ret_5d", "GILD_Gilead_ret_20d", "NWL_Newell_ret_20d", "ORCL_zscore_60d", "EWY_Korea_zscore_60d", "EWL_Switzerland_vol_20d", "EWA_Australia_zscore_60d", "HangSeng_HK_ret_1d", "3M_vol_20d", "US3M_Rate_zscore_60d", "PCAR_PaccarInc_ret_5d", "BA_ret_1d", "ITT_ITTInc_ret_5d", "Industrial_Production_zscore_60d", "AORD_AUS_zscore_60d"], "is_new": true}, {"model_id": "new_h5_CALM_LogisticRegression_N20_t6", "algo": "LogisticRegression", "regime": "CALM", "horizon": 5, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "HD_zscore_60d", "EWM_Malaysia_vol_20d", "CLX_Clorox_vol_20d", "PPL_PPL_ret_1d", "DE_Deere_vol_20d", "SBUX_vol_20d", "HD_ret_5d", "XOM_ret_1d", "NWL_Newell_ret_20d", "EWM_Malaysia_ret_1d", "CPB_CampbellSoup_ret_20d", "SJM_JM_Smucker_ret_5d", "SLB_Schlumberger_ret_1d", "LOW_Lowes_ret_20d", "EWQ_France_ret_20d", "EXC_Exelon_ret_1d", "AMT_AmericanTower_ret_1d", "DIS_vol_20d"], "is_new": true}, {"model_id": "new_h5_CALM_LogisticRegression_N20_t7", "algo": "LogisticRegression", "regime": "CALM", "horizon": 5, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "IYM_BasicMaterials_ret_20d", "T10Y2Y_Spread_ret_5d", "INTC_ret_5d", "DOW_Price_zscore_60d", "MSTR_Bitcoin3_ret_20d", "EWH_HongKong_ret_5d", "EOG_EOGResources_ret_5d", "CI_Cigna_vol_20d", "AXP_Amex_ret_20d", "PAYX_Paychex_ret_20d", "XOM_ret_20d", "Industrial_Production_zscore_60d", "PPL_PPL_ret_1d", "QQQ_vol_20d", "SO_SouthernCo_ret_5d", "T_ret_1d", "HD_ret_20d", "NVDA_vol_20d"], "is_new": true}, {"model_id": "new_h5_CALM_LogisticRegression_N25_t0", "algo": "LogisticRegression", "regime": "CALM", "horizon": 5, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "DE_Deere_vol_20d", "AMGN_Amgen_ret_1d", "CLX_Clorox_vol_20d", "CMCSA_ret_1d", "SPY_zscore_60d", "NVDA_vol_20d", "MS_MorganStanley_ret_5d", "SLB_Schlumberger_ret_5d", "EOG_EOGResources_vol_20d", "PPL_PPL_ret_1d", "ENB_EnbridgeInc_ret_1d", "M_Macys_vol_20d", "EWM_Malaysia_vol_20d", "HangSeng_HK_ret_5d", "AXP_Amex_ret_20d", "CCI_CrownCastle_vol_20d", "heston_var_ev_h5", "TXN_vol_20d", "spx_momentum_3d", "HD_ret_20d", "US3M_Rate_vol_20d", "ORCL_zscore_60d", "VOD_Vodafone_zscore_60d"], "is_new": true}, {"model_id": "new_h5_CALM_LogisticRegression_N25_t1", "algo": "LogisticRegression", "regime": "CALM", "horizon": 5, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "SJM_JM_Smucker_ret_5d", "DAX_Germany_zscore_60d", "PG_ret_20d", "CCI_CrownCastle_vol_20d", "heston_var_ev_h7", "LOW_Lowes_ret_20d", "CPB_CampbellSoup_vol_20d", "EWA_Australia_zscore_60d", "CTAS_Cintas_vol_20d", "VRP_ma5", "GE_ret_1d", "TED_Spread_vol_20d", "EOG_EOGResources_ret_5d", "TM_Telephone_ret_1d", "US30Y_Rate_ret_20d", "DHR_ret_1d", "PLD_Prologis_ret_5d", "NOC_Northrop_ret_20d", "EWL_Switzerland_zscore_60d", "gjr_condvar_h1", "EWH_HongKong_ret_5d", "FedFunds_zscore_60d", "Nikkei_Japan_zscore_60d"], "is_new": true}, {"model_id": "new_h5_CALM_LogisticRegression_N25_t2", "algo": "LogisticRegression", "regime": "CALM", "horizon": 5, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "TGT_Target_zscore_60d", "TED_Spread_zscore_60d", "M_Macys_vol_20d", "heston_var_ev_h3", "TM_Telephone_ret_1d", "EWC_Canada_zscore_60d", "INTC_ret_5d", "HD_ret_1d", "PPL_PPL_ret_1d", "EWM_Malaysia_zscore_60d", "PLD_Prologis_ret_5d", "HD_ret_5d", "IYR_US_REIT2_zscore_60d", "PFE_ret_1d", "CTAS_Cintas_vol_20d", "IBEX_Spain_ret_20d", "PAYX_Paychex_vol_20d", "ES_Evergy_ret_1d", "XLF_Fin_vol_20d", "FedFunds_zscore_60d", "PCAR_PaccarInc_ret_5d", "EWG_Germany_ret_20d", "DAX_Germany_vol_20d"], "is_new": true}, {"model_id": "new_h5_CALM_LogisticRegression_N25_t3", "algo": "LogisticRegression", "regime": "CALM", "horizon": 5, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "DAX_Germany_zscore_60d", "EWY_Korea_ret_20d", "EWA_Australia_ret_1d", "NWL_Newell_ret_20d", "PAYX_Paychex_ret_20d", "SO_SouthernCo_ret_5d", "CPB_CampbellSoup_vol_20d", "DIS_vol_20d", "CPB_CampbellSoup_zscore_60d", "IYM_BasicMaterials_ret_20d", "LMT_LockheedMartin_vol_20d", "GE_ret_1d", "EFFR_vol_20d", "EWL_Switzerland_zscore_60d", "ORCL_zscore_60d", "EWS_Singapore_ret_5d", "VOD_Vodafone_zscore_60d", "VRP_ma5", "PAYX_Paychex_zscore_60d", "WTI_Oil_FRED_zscore_60d", "AMT_AmericanTower_ret_1d", "XOM_ret_20d", "heston_var_ev_h7"], "is_new": true}, {"model_id": "new_h5_CALM_LogisticRegression_N25_t4", "algo": "LogisticRegression", "regime": "CALM", "horizon": 5, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWL_Switzerland_vol_20d", "MS_MorganStanley_zscore_60d", "GE_ret_1d", "EWA_Australia_zscore_60d", "DOW_Price_zscore_60d", "EWS_Singapore_ret_5d", "MO_AltriaMG_ret_1d", "US3M_Rate_vol_20d", "ASX_Australia_ret_5d", "M_Macys_vol_20d", "hmm_p_stress", "DAX_Germany_zscore_60d", "SLB_Schlumberger_ret_1d", "PFE_ret_1d", "PAYX_Paychex_zscore_60d", "ITT_ITTInc_ret_5d", "Industrial_Production_zscore_60d", "EXC_Exelon_zscore_60d", "SJM_JM_Smucker_ret_5d", "HD_ret_5d", "ASX_Australia_vol_20d", "AMD_ret_5d", "HD_ret_20d"], "is_new": true}, {"model_id": "new_h5_CALM_LogisticRegression_N25_t5", "algo": "LogisticRegression", "regime": "CALM", "horizon": 5, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "VRP_ma5", "T10Y2Y_Spread_ret_5d", "SPY_zscore_60d", "IYR_US_REIT2_zscore_60d", "PFE_ret_1d", "NEE_NextEra_ret_20d", "3M_vol_20d", "BTI_BritishAmerican_ret_5d", "US3Y_Rate_ret_5d", "Michigan_Sentiment_ret_20d", "NOC_Northrop_ret_20d", "MS_MorganStanley_ret_1d", "PAYX_Paychex_ret_20d", "heston_var_ev_h3", "EWG_Germany_vol_20d", "US6M_Rate_ret_20d", "US3M_Rate_vol_20d", "ES_Evergy_ret_1d", "SCHW_Schwab_ret_5d", "TM_Telephone_ret_1d", "EWA_Australia_zscore_60d", "XOM_ret_20d", "ASX_Australia_ret_5d"], "is_new": true}, {"model_id": "new_h5_CALM_LogisticRegression_N25_t6", "algo": "LogisticRegression", "regime": "CALM", "horizon": 5, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "TGT_Target_zscore_60d", "WTI_Oil_FRED_zscore_60d", "gjr_condvar_h1", "EQR_Equity_ret_1d", "AMGN_Amgen_ret_1d", "SJM_JM_Smucker_ret_1d", "US5Y_Rate_ret_5d", "SLB_Schlumberger_ret_5d", "EWQ_France_ret_20d", "EWS_Singapore_ret_5d", "T10Y2Y_Spread_ret_5d", "Brent_Oil_FRED_ret_20d", "AMD_ret_5d", "CPB_CampbellSoup_zscore_60d", "EWA_Australia_ret_1d", "Michigan_Sentiment_ret_20d", "DE_Deere_ret_5d", "HD_ret_1d", "3M_vol_20d", "SLB_Schlumberger_ret_1d", "XLV_Health_zscore_60d", "CCI_CrownCastle_vol_20d", "IYR_US_REIT2_zscore_60d"], "is_new": true}, {"model_id": "new_h5_CALM_LogisticRegression_N25_t7", "algo": "LogisticRegression", "regime": "CALM", "horizon": 5, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "MS_MorganStanley_zscore_60d", "SBUX_ret_5d", "HD_ret_5d", "EWY_Korea_zscore_60d", "ITT_ITTInc_ret_5d", "NFCI_ret_5d", "EWA_Australia_zscore_60d", "EQIX_Equinix_ret_5d", "DIS_vol_20d", "IYM_BasicMaterials_ret_20d", "AMT_AmericanTower_ret_1d", "LUV_SouthwestAir_ret_5d", "HangSeng_HK_vol_20d", "MSTR_Bitcoin3_ret_20d", "PLD_Prologis_ret_5d", "T10Y2Y_Spread_ret_5d", "XLK_Tech_zscore_60d", "spx_vol_5d", "SJM_JM_Smucker_ret_5d", "US1Y_Rate_ret_5d", "AMD_ret_1d", "MRK_Merck_zscore_60d", "US5Y_Rate_ret_5d"], "is_new": true}, {"model_id": "new_h5_CALM_LogisticRegression_N30_t0", "algo": "LogisticRegression", "regime": "CALM", "horizon": 5, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EQR_Equity_ret_1d", "3M_vol_20d", "AORD_AUS_zscore_60d", "DAX_Germany_zscore_60d", "TGT_Target_zscore_60d", "US3M_Rate_zscore_60d", "DOW_Price_zscore_60d", "EWM_Malaysia_ret_1d", "HangSeng_HK_ret_5d", "EWL_Switzerland_zscore_60d", "PAYX_Paychex_zscore_60d", "CPB_CampbellSoup_ret_20d", "HUM_Humana_ret_5d", "EOG_EOGResources_vol_20d", "HD_ret_20d", "EWA_Australia_zscore_60d", "SJM_JM_Smucker_ret_5d", "SO_SouthernCo_ret_5d", "BLK_BlackRock_zscore_60d", "DAX_Germany_vol_20d", "EWS_Singapore_ret_5d", "PG_ret_20d", "EQIX_Equinix_ret_5d", "GE_ret_1d", "EFFR_vol_20d", "TM_Telephone_ret_1d", "US1Y_Rate_ret_20d", "XOM_ret_1d"], "is_new": true}, {"model_id": "new_h5_CALM_LogisticRegression_N30_t1", "algo": "LogisticRegression", "regime": "CALM", "horizon": 5, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "SBUX_zscore_60d", "EWM_Malaysia_ret_1d", "TED_Spread_zscore_60d", "Industrial_Production_zscore_60d", "EWM_Malaysia_zscore_60d", "XOM_ret_20d", "PLD_Prologis_ret_5d", "INTC_ret_1d", "CPB_CampbellSoup_zscore_60d", "EQIX_Equinix_ret_5d", "DOW_Price_zscore_60d", "AMD_ret_5d", "SBUX_vol_20d", "DHR_ret_1d", "CI_Cigna_vol_20d", "INTC_ret_5d", "US3Y_Rate_ret_5d", "TXN_vol_20d", "AXP_Amex_vol_20d", "IYM_BasicMaterials_ret_20d", "MSTR_Bitcoin3_ret_5d", "3M_vol_20d", "heston_ev_h3", "ES_Evergy_ret_1d", "gjr_condvar_h1", "GD_GeneralDynamics_zscore_60d", "SCHW_Schwab_ret_5d", "ASX_Australia_vol_20d"], "is_new": true}, {"model_id": "new_h5_CALM_LogisticRegression_N30_t2", "algo": "LogisticRegression", "regime": "CALM", "horizon": 5, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWA_Australia_ret_1d", "PG_ret_20d", "TGT_Target_zscore_60d", "AMGN_Amgen_ret_1d", "BLK_BlackRock_zscore_60d", "MS_MorganStanley_ret_1d", "TM_Telephone_ret_1d", "IBEX_Spain_ret_20d", "AMD_ret_1d", "XLV_Health_zscore_60d", "NEE_NextEra_ret_20d", "HD_ret_1d", "DIS_vol_20d", "US3Y_Rate_ret_5d", "EWM_Malaysia_zscore_60d", "US30Y_Rate_ret_20d", "LOW_Lowes_ret_5d", "IWM_SmallCap_vol_20d", "LUV_SouthwestAir_ret_5d", "AVB_AvalonBay_zscore_60d", "ORCL_zscore_60d", "VOD_Vodafone_zscore_60d", "vix_mean_abs_ret_5d", "EOG_EOGResources_vol_20d", "3M_vol_20d", "ASX_Australia_vol_20d", "TED_Spread_zscore_60d", "DE_Deere_vol_20d"], "is_new": true}, {"model_id": "new_h5_CALM_LogisticRegression_N30_t3", "algo": "LogisticRegression", "regime": "CALM", "horizon": 5, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "VRP_ma5", "DAX_Germany_vol_20d", "EWL_Switzerland_zscore_60d", "AVB_AvalonBay_zscore_60d", "XOM_ret_20d", "MSTR_Bitcoin3_ret_5d", "CPB_CampbellSoup_ret_5d", "3M_ret_5d", "SO_SouthernCo_ret_5d", "MO_AltriaMG_ret_1d", "INTC_ret_5d", "SBUX_vol_20d", "BDX_Becton_Dickinson_ret_20d", "HD_ret_1d", "vix_acceleration_1d", "US6M_Rate_ret_20d", "EWA_Australia_ret_1d", "NEE_NextEra_ret_20d", "EXC_Exelon_ret_1d", "heston_var_ev_h5", "MSTR_Bitcoin3_ret_20d", "GILD_Gilead_ret_20d", "heston_var_ev_h7", "MRK_Merck_zscore_60d", "CPB_CampbellSoup_zscore_60d", "XLV_Health_zscore_60d", "SLB_Schlumberger_ret_5d", "DE_Deere_vol_20d"], "is_new": true}, {"model_id": "new_h5_CALM_LogisticRegression_N30_t4", "algo": "LogisticRegression", "regime": "CALM", "horizon": 5, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWM_Malaysia_ret_1d", "ENB_EnbridgeInc_ret_1d", "US5Y_Rate_ret_5d", "MRK_Merck_zscore_60d", "BLK_BlackRock_zscore_60d", "Brent_Oil_FRED_ret_5d", "AMZN_ret_5d", "DAX_Germany_zscore_60d", "EWS_Singapore_ret_5d", "EQIX_Equinix_ret_5d", "EWA_Australia_zscore_60d", "EFFR_vol_20d", "Core_PCE_zscore_60d", "EWL_Switzerland_vol_20d", "US3M_Rate_zscore_60d", "spx_vol_5d", "DE_Deere_vol_20d", "TM_Telephone_vol_20d", "INTC_ret_1d", "SLB_Schlumberger_ret_1d", "XOM_ret_1d", "HD_ret_20d", "ES_Evergy_ret_1d", "SCHW_Schwab_ret_5d", "EWM_Malaysia_zscore_60d", "INTC_ret_5d", "ASX_Australia_vol_20d", "EOG_EOGResources_vol_20d"], "is_new": true}, {"model_id": "new_h5_CALM_LogisticRegression_N30_t5", "algo": "LogisticRegression", "regime": "CALM", "horizon": 5, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "MS_MorganStanley_ret_5d", "CMCSA_ret_1d", "BA_ret_1d", "CI_Cigna_vol_20d", "WTI_Oil_FRED_zscore_60d", "US6M_Rate_ret_20d", "Industrial_Production_zscore_60d", "EWM_Malaysia_ret_1d", "DHR_vol_20d", "vix_acceleration_1d", "3M_ret_5d", "IYM_BasicMaterials_ret_20d", "XLK_Tech_zscore_60d", "Retail_Sales_zscore_60d", "SJM_JM_Smucker_ret_5d", "HangSeng_HK_ret_1d", "EQIX_Equinix_ret_5d", "TM_Telephone_vol_20d", "CLX_Clorox_vol_20d", "HUM_Humana_ret_5d", "EWY_Korea_ret_20d", "ITT_ITTInc_ret_5d", "Core_PCE_zscore_60d", "HangSeng_HK_ret_5d", "DAX_Germany_vol_20d", "US3Y_Rate_ret_5d", "PLD_Prologis_ret_5d", "TM_Telephone_ret_1d"], "is_new": true}, {"model_id": "new_h5_CALM_LogisticRegression_N30_t6", "algo": "LogisticRegression", "regime": "CALM", "horizon": 5, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "AMT_AmericanTower_ret_1d", "vix_acceleration_1d", "CPB_CampbellSoup_zscore_60d", "VVIX_ret_20d", "spx_abs_ret_max_5d", "CPB_CampbellSoup_vol_20d", "spx_vol_5d", "LMT_LockheedMartin_ret_1d", "GILD_Gilead_ret_20d", "EWH_HongKong_ret_5d", "Core_CPI_zscore_60d", "ORCL_vol_20d", "PAYX_Paychex_vol_20d", "BA_ret_1d", "CCI_CrownCastle_vol_20d", "HD_zscore_60d", "CTAS_Cintas_vol_20d", "3M_ret_5d", "HangSeng_HK_ret_5d", "US30Y_Rate_ret_20d", "T10Y2Y_Spread_ret_5d", "DHR_ret_1d", "AORD_AUS_zscore_60d", "NEE_NextEra_ret_20d", "FedFunds_zscore_60d", "heston_var_ev_h5", "MSTR_Bitcoin3_ret_20d", "DIS_vol_20d"], "is_new": true}, {"model_id": "new_h5_CALM_LogisticRegression_N30_t7", "algo": "LogisticRegression", "regime": "CALM", "horizon": 5, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "gjr_condvar_h1", "EQIX_Equinix_ret_5d", "SCHW_Schwab_ret_5d", "EWA_Australia_ret_1d", "Nikkei_Japan_zscore_60d", "MO_AltriaMG_ret_1d", "DIS_vol_20d", "US3M_Rate_vol_20d", "EWG_Germany_vol_20d", "XLK_Tech_zscore_60d", "PCAR_PaccarInc_ret_5d", "DE_Deere_ret_5d", "MRK_Merck_zscore_60d", "DAX_Germany_zscore_60d", "BA_ret_1d", "INTC_ret_1d", "MSTR_Bitcoin3_ret_1d", "LMT_LockheedMartin_ret_1d", "NFCI_ret_5d", "PG_ret_20d", "CPB_CampbellSoup_ret_20d", "NWL_Newell_ret_20d", "PAYX_Paychex_ret_20d", "Brent_Oil_FRED_ret_20d", "DHR_vol_20d", "VVIX_ret_20d", "IBEX_Spain_ret_20d", "GE_ret_1d"], "is_new": true}, {"model_id": "new_h5_NORMAL_XGBoost_N5_t0", "algo": "XGBoost", "regime": "NORMAL", "horizon": 5, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "AMT_AmericanTower_ret_1d", "LMT_LockheedMartin_ret_1d", "US5Y_Rate_ret_5d"], "is_new": true}, {"model_id": "new_h5_NORMAL_XGBoost_N5_t1", "algo": "XGBoost", "regime": "NORMAL", "horizon": 5, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWY_Korea_ret_20d", "hmm_p_stress", "ENB_EnbridgeInc_ret_1d"], "is_new": true}, {"model_id": "new_h5_NORMAL_XGBoost_N5_t2", "algo": "XGBoost", "regime": "NORMAL", "horizon": 5, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "vix_acceleration_1d", "EWC_Canada_zscore_60d", "AMGN_Amgen_ret_1d"], "is_new": true}, {"model_id": "new_h5_NORMAL_XGBoost_N5_t3", "algo": "XGBoost", "regime": "NORMAL", "horizon": 5, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "Core_CPI_zscore_60d", "US5Y_Rate_ret_5d", "EQR_Equity_ret_1d"], "is_new": true}, {"model_id": "new_h5_NORMAL_XGBoost_N5_t4", "algo": "XGBoost", "regime": "NORMAL", "horizon": 5, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "HD_ret_5d", "AXP_Amex_ret_20d", "GD_GeneralDynamics_zscore_60d"], "is_new": true}, {"model_id": "new_h5_NORMAL_XGBoost_N5_t5", "algo": "XGBoost", "regime": "NORMAL", "horizon": 5, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "Nikkei_Japan_vol_20d", "MS_MorganStanley_ret_1d", "T10Y2Y_Spread_ret_5d"], "is_new": true}, {"model_id": "new_h5_NORMAL_XGBoost_N5_t6", "algo": "XGBoost", "regime": "NORMAL", "horizon": 5, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "CPB_CampbellSoup_ret_5d", "INTC_ret_5d", "Nikkei_Japan_zscore_60d"], "is_new": true}, {"model_id": "new_h5_NORMAL_XGBoost_N5_t7", "algo": "XGBoost", "regime": "NORMAL", "horizon": 5, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "TM_Telephone_vol_20d", "FedFunds_zscore_60d", "SPY_zscore_60d"], "is_new": true}, {"model_id": "new_h5_NORMAL_XGBoost_N8_t0", "algo": "XGBoost", "regime": "NORMAL", "horizon": 5, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "HD_zscore_60d", "IYR_US_REIT2_zscore_60d", "heston_var_ev_h7", "XLK_Tech_zscore_60d", "TXN_vol_20d", "ES_Evergy_ret_1d"], "is_new": true}, {"model_id": "new_h5_NORMAL_XGBoost_N8_t1", "algo": "XGBoost", "regime": "NORMAL", "horizon": 5, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "QQQ_vol_20d", "ASX_Australia_vol_20d", "JNJ_ret_1d", "spx_vol_5d", "ORCL_vol_20d", "HangSeng_HK_ret_5d"], "is_new": true}, {"model_id": "new_h5_NORMAL_XGBoost_N8_t2", "algo": "XGBoost", "regime": "NORMAL", "horizon": 5, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "DIS_vol_20d", "US6M_Rate_ret_20d", "DAX_Germany_vol_20d", "EOG_EOGResources_ret_5d", "M_Macys_vol_20d", "US3M_Rate_vol_20d"], "is_new": true}, {"model_id": "new_h5_NORMAL_XGBoost_N8_t3", "algo": "XGBoost", "regime": "NORMAL", "horizon": 5, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWM_Malaysia_ret_1d", "GE_ret_1d", "LOW_Lowes_ret_20d", "EWA_Australia_zscore_60d", "TXN_vol_20d", "HangSeng_HK_ret_1d"], "is_new": true}, {"model_id": "new_h5_NORMAL_XGBoost_N8_t4", "algo": "XGBoost", "regime": "NORMAL", "horizon": 5, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "WTI_Oil_FRED_zscore_60d", "PCAR_PaccarInc_ret_5d", "SLB_Schlumberger_ret_1d", "SLB_Schlumberger_ret_5d", "hmm_p_stress", "EWC_Canada_zscore_60d"], "is_new": true}, {"model_id": "new_h5_NORMAL_XGBoost_N8_t5", "algo": "XGBoost", "regime": "NORMAL", "horizon": 5, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "AXP_Amex_vol_20d", "SJM_JM_Smucker_ret_5d", "JNJ_ret_1d", "EFFR_ret_1d", "AXP_Amex_ret_20d", "heston_var_ev_h5"], "is_new": true}, {"model_id": "new_h5_NORMAL_XGBoost_N8_t6", "algo": "XGBoost", "regime": "NORMAL", "horizon": 5, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "XLV_Health_zscore_60d", "TED_Spread_vol_20d", "AMD_ret_1d", "EWA_Australia_ret_1d", "EXC_Exelon_zscore_60d", "EXC_Exelon_ret_1d"], "is_new": true}, {"model_id": "new_h5_NORMAL_XGBoost_N8_t7", "algo": "XGBoost", "regime": "NORMAL", "horizon": 5, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "LOW_Lowes_ret_20d", "XLV_Health_zscore_60d", "LUV_SouthwestAir_ret_5d", "CCI_CrownCastle_vol_20d", "US7Y_Rate_ret_20d", "PPL_PPL_ret_1d"], "is_new": true}, {"model_id": "new_h5_NORMAL_XGBoost_N10_t0", "algo": "XGBoost", "regime": "NORMAL", "horizon": 5, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "ASX_Australia_ret_5d", "EWY_Korea_zscore_60d", "HUM_Humana_ret_5d", "vix_mean_abs_ret_5d", "heston_ev_h3", "vix_acceleration_1d", "HangSeng_HK_ret_5d", "US3M_Rate_vol_20d"], "is_new": true}, {"model_id": "new_h5_NORMAL_XGBoost_N10_t1", "algo": "XGBoost", "regime": "NORMAL", "horizon": 5, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "HD_ret_1d", "INTC_ret_5d", "LUV_SouthwestAir_ret_5d", "GE_ret_1d", "XLB_Materials_zscore_60d", "CLX_Clorox_vol_20d", "EQR_Equity_ret_1d", "TED_Spread_vol_20d"], "is_new": true}, {"model_id": "new_h5_NORMAL_XGBoost_N10_t2", "algo": "XGBoost", "regime": "NORMAL", "horizon": 5, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "US3Y_Rate_ret_5d", "HD_ret_20d", "US7Y_Rate_ret_20d", "HUM_Humana_ret_5d", "GE_ret_1d", "DE_Deere_ret_5d", "ES_Evergy_ret_1d", "SO_SouthernCo_ret_5d"], "is_new": true}, {"model_id": "new_h5_NORMAL_XGBoost_N10_t3", "algo": "XGBoost", "regime": "NORMAL", "horizon": 5, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "HUM_Humana_ret_5d", "EFFR_ret_1d", "ORCL_zscore_60d", "WTI_Oil_FRED_zscore_60d", "NFCI_ret_5d", "EWL_Switzerland_zscore_60d", "3M_ret_5d", "HD_ret_1d"], "is_new": true}, {"model_id": "new_h5_NORMAL_XGBoost_N10_t4", "algo": "XGBoost", "regime": "NORMAL", "horizon": 5, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "US3M_Rate_zscore_60d", "HangSeng_HK_ret_5d", "PAYX_Paychex_ret_20d", "TGT_Target_zscore_60d", "hmm_p_stress", "MSTR_Bitcoin3_ret_20d", "HD_ret_20d", "BTI_BritishAmerican_ret_5d"], "is_new": true}, {"model_id": "new_h5_NORMAL_XGBoost_N10_t5", "algo": "XGBoost", "regime": "NORMAL", "horizon": 5, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "AMD_ret_5d", "Industrial_Production_zscore_60d", "US3M_Rate_vol_20d", "EMR_Emerson_ret_20d", "LMT_LockheedMartin_ret_1d", "CPB_CampbellSoup_vol_20d", "ENB_EnbridgeInc_ret_1d", "Core_PCE_zscore_60d"], "is_new": true}, {"model_id": "new_h5_NORMAL_XGBoost_N10_t6", "algo": "XGBoost", "regime": "NORMAL", "horizon": 5, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWS_Singapore_ret_5d", "CPB_CampbellSoup_vol_20d", "CI_Cigna_vol_20d", "ORCL_vol_20d", "MSTR_Bitcoin3_ret_1d", "MO_AltriaMG_ret_1d", "MS_MorganStanley_ret_1d", "EWM_Malaysia_ret_1d"], "is_new": true}, {"model_id": "new_h5_NORMAL_XGBoost_N10_t7", "algo": "XGBoost", "regime": "NORMAL", "horizon": 5, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "AXP_Amex_vol_20d", "HUM_Humana_ret_5d", "EWA_Australia_zscore_60d", "EWG_Germany_ret_20d", "INTC_ret_5d", "US6M_Rate_ret_20d", "AVB_AvalonBay_zscore_60d", "CCI_CrownCastle_vol_20d"], "is_new": true}, {"model_id": "new_h5_NORMAL_XGBoost_N12_t0", "algo": "XGBoost", "regime": "NORMAL", "horizon": 5, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "PFE_ret_1d", "vix_acceleration_1d", "gjr_condvar_h1", "LUV_SouthwestAir_ret_5d", "IYM_BasicMaterials_ret_20d", "Michigan_Sentiment_ret_20d", "US3M_Rate_zscore_60d", "HD_ret_5d", "CMCSA_ret_1d", "FedFunds_zscore_60d"], "is_new": true}, {"model_id": "new_h5_NORMAL_XGBoost_N12_t1", "algo": "XGBoost", "regime": "NORMAL", "horizon": 5, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "XLY_Disc_vol_20d", "DIS_vol_20d", "SLB_Schlumberger_ret_1d", "EWA_Australia_zscore_60d", "XLV_Health_zscore_60d", "MSTR_Bitcoin3_ret_5d", "DOW_Price_zscore_60d", "MO_AltriaMG_ret_1d", "QQQ_vol_20d", "PFE_ret_1d"], "is_new": true}, {"model_id": "new_h5_NORMAL_XGBoost_N12_t2", "algo": "XGBoost", "regime": "NORMAL", "horizon": 5, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "NWL_Newell_ret_20d", "LUV_SouthwestAir_ret_5d", "NOC_Northrop_ret_20d", "MS_MorganStanley_ret_5d", "INTC_ret_1d", "PG_ret_20d", "M_Macys_vol_20d", "IBEX_Spain_ret_20d", "AMD_ret_5d", "MS_MorganStanley_zscore_60d"], "is_new": true}, {"model_id": "new_h5_NORMAL_XGBoost_N12_t3", "algo": "XGBoost", "regime": "NORMAL", "horizon": 5, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "JNJ_ret_1d", "SLB_Schlumberger_ret_5d", "MS_MorganStanley_ret_5d", "Core_CPI_zscore_60d", "PAYX_Paychex_vol_20d", "CI_Cigna_vol_20d", "BDX_Becton_Dickinson_ret_20d", "NWL_Newell_ret_20d", "US3Y_Rate_ret_5d", "NEE_NextEra_ret_20d"], "is_new": true}, {"model_id": "new_h5_NORMAL_XGBoost_N12_t4", "algo": "XGBoost", "regime": "NORMAL", "horizon": 5, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "HD_ret_1d", "US7Y_Rate_ret_20d", "heston_var_ev_h5", "NEE_NextEra_ret_20d", "EFFR_ret_1d", "EWM_Malaysia_vol_20d", "IBEX_Spain_ret_20d", "HD_ret_20d", "EWG_Germany_ret_20d", "NWL_Newell_ret_20d"], "is_new": true}, {"model_id": "new_h5_NORMAL_XGBoost_N12_t5", "algo": "XGBoost", "regime": "NORMAL", "horizon": 5, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "LLY_zscore_60d", "NOC_Northrop_ret_20d", "ES_Evergy_ret_1d", "TED_Spread_vol_20d", "EWG_Germany_vol_20d", "SJM_JM_Smucker_ret_1d", "WTI_Oil_FRED_zscore_60d", "HD_ret_20d", "PAYX_Paychex_ret_20d", "PAYX_Paychex_vol_20d"], "is_new": true}, {"model_id": "new_h5_NORMAL_XGBoost_N12_t6", "algo": "XGBoost", "regime": "NORMAL", "horizon": 5, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWL_Switzerland_vol_20d", "EWM_Malaysia_ret_1d", "EWY_Korea_ret_20d", "NOC_Northrop_ret_20d", "BDX_Becton_Dickinson_ret_20d", "Core_CPI_zscore_60d", "BA_ret_1d", "PPL_PPL_ret_1d", "TM_Telephone_ret_1d", "NWL_Newell_ret_20d"], "is_new": true}, {"model_id": "new_h5_NORMAL_XGBoost_N12_t7", "algo": "XGBoost", "regime": "NORMAL", "horizon": 5, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWJ_Japan_vol_20d", "HD_ret_20d", "IYR_US_REIT2_zscore_60d", "ASX_Australia_vol_20d", "EWG_Germany_vol_20d", "IYM_BasicMaterials_ret_20d", "T_ret_1d", "CPB_CampbellSoup_ret_5d", "Nikkei_Japan_vol_20d", "spx_vol_5d"], "is_new": true}, {"model_id": "new_h5_NORMAL_XGBoost_N15_t0", "algo": "XGBoost", "regime": "NORMAL", "horizon": 5, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "XOM_ret_20d", "MS_MorganStanley_ret_5d", "CPB_CampbellSoup_zscore_60d", "SLB_Schlumberger_ret_5d", "IBEX_Spain_ret_20d", "MSTR_Bitcoin3_ret_20d", "HangSeng_HK_ret_5d", "US1Y_Rate_ret_20d", "SCHW_Schwab_ret_5d", "MSTR_Bitcoin3_ret_1d", "3M_vol_20d", "T10Y2Y_Spread_ret_5d", "vix_mean_abs_ret_5d"], "is_new": true}, {"model_id": "new_h5_NORMAL_XGBoost_N15_t1", "algo": "XGBoost", "regime": "NORMAL", "horizon": 5, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "Nikkei_Japan_vol_20d", "AMZN_ret_5d", "EWG_Germany_vol_20d", "MSTR_Bitcoin3_ret_1d", "EWG_Germany_ret_20d", "spx_vol_5d", "EOG_EOGResources_ret_5d", "HD_ret_20d", "DE_Deere_ret_5d", "TM_Telephone_ret_1d", "Nikkei_Japan_zscore_60d", "Michigan_Sentiment_ret_20d", "EQIX_Equinix_ret_5d"], "is_new": true}, {"model_id": "new_h5_NORMAL_XGBoost_N15_t2", "algo": "XGBoost", "regime": "NORMAL", "horizon": 5, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "CLX_Clorox_vol_20d", "PCAR_PaccarInc_ret_5d", "EWM_Malaysia_ret_1d", "NEE_NextEra_ret_20d", "MS_MorganStanley_ret_1d", "Industrial_Production_zscore_60d", "ORCL_vol_20d", "NVDA_vol_20d", "PFE_ret_1d", "EWM_Malaysia_zscore_60d", "XLF_Fin_vol_20d", "BDX_Becton_Dickinson_ret_20d", "MSTR_Bitcoin3_ret_1d"], "is_new": true}, {"model_id": "new_h5_NORMAL_XGBoost_N15_t3", "algo": "XGBoost", "regime": "NORMAL", "horizon": 5, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "SJM_JM_Smucker_ret_1d", "SCHW_Schwab_ret_5d", "EMR_Emerson_ret_20d", "vix_acceleration_1d", "EWA_Australia_zscore_60d", "NFCI_ret_5d", "MRK_Merck_zscore_60d", "PAYX_Paychex_vol_20d", "Brent_Oil_FRED_ret_5d", "EXC_Exelon_zscore_60d", "PFE_ret_1d", "Nikkei_Japan_zscore_60d", "LMT_LockheedMartin_vol_20d"], "is_new": true}, {"model_id": "new_h5_NORMAL_XGBoost_N15_t4", "algo": "XGBoost", "regime": "NORMAL", "horizon": 5, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EXC_Exelon_ret_1d", "AORD_AUS_zscore_60d", "EFFR_vol_20d", "MS_MorganStanley_zscore_60d", "CTAS_Cintas_vol_20d", "spx_momentum_3d", "DAX_Germany_zscore_60d", "EWC_Canada_zscore_60d", "US7Y_Rate_ret_20d", "Industrial_Production_zscore_60d", "PAYX_Paychex_vol_20d", "EOG_EOGResources_vol_20d", "GD_GeneralDynamics_zscore_60d"], "is_new": true}, {"model_id": "new_h5_NORMAL_XGBoost_N15_t5", "algo": "XGBoost", "regime": "NORMAL", "horizon": 5, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "Nikkei_Japan_zscore_60d", "GE_ret_1d", "TM_Telephone_ret_1d", "ES_Evergy_ret_1d", "NWL_Newell_ret_20d", "CPB_CampbellSoup_ret_20d", "HangSeng_HK_ret_5d", "TED_Spread_vol_20d", "heston_ev_h3", "Brent_Oil_FRED_ret_20d", "AMZN_ret_5d", "US6M_Rate_ret_20d", "VVIX_ret_20d"], "is_new": true}, {"model_id": "new_h5_NORMAL_XGBoost_N15_t6", "algo": "XGBoost", "regime": "NORMAL", "horizon": 5, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWG_Germany_vol_20d", "SPY_zscore_60d", "heston_var_ev_h3", "AMGN_Amgen_ret_1d", "SBUX_zscore_60d", "AMD_ret_5d", "EWL_Switzerland_zscore_60d", "gjr_condvar_h1", "EWQ_France_zscore_60d", "Core_CPI_zscore_60d", "LMT_LockheedMartin_ret_1d", "MS_MorganStanley_ret_5d", "NOC_Northrop_ret_20d"], "is_new": true}, {"model_id": "new_h5_NORMAL_XGBoost_N15_t7", "algo": "XGBoost", "regime": "NORMAL", "horizon": 5, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWY_Korea_ret_20d", "heston_var_ev_h7", "vix_mean_abs_ret_5d", "AVB_AvalonBay_zscore_60d", "TGT_Target_zscore_60d", "IYM_BasicMaterials_ret_20d", "CLX_Clorox_vol_20d", "WTI_Oil_FRED_zscore_60d", "TM_Telephone_ret_1d", "EWG_Germany_ret_20d", "TXN_vol_20d", "MSTR_Bitcoin3_ret_5d", "BDX_Becton_Dickinson_ret_20d"], "is_new": true}, {"model_id": "new_h5_NORMAL_XGBoost_N20_t0", "algo": "XGBoost", "regime": "NORMAL", "horizon": 5, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "XLY_Disc_vol_20d", "T_ret_1d", "XLB_Materials_zscore_60d", "LUV_SouthwestAir_ret_5d", "NOC_Northrop_ret_20d", "US3Y_Rate_ret_5d", "CLX_Clorox_vol_20d", "BTI_BritishAmerican_ret_20d", "AVB_AvalonBay_zscore_60d", "EWA_Australia_zscore_60d", "AMD_ret_5d", "TM_Telephone_vol_20d", "VOD_Vodafone_zscore_60d", "ORCL_zscore_60d", "LOW_Lowes_ret_20d", "TED_Spread_zscore_60d", "MSTR_Bitcoin3_ret_5d", "EWJ_Japan_vol_20d"], "is_new": true}, {"model_id": "new_h5_NORMAL_XGBoost_N20_t1", "algo": "XGBoost", "regime": "NORMAL", "horizon": 5, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "HUM_Humana_ret_5d", "GD_GeneralDynamics_zscore_60d", "CPB_CampbellSoup_ret_5d", "MS_MorganStanley_ret_1d", "Brent_Oil_FRED_ret_5d", "MS_MorganStanley_zscore_60d", "Michigan_Sentiment_ret_20d", "heston_ev_h3", "heston_var_ev_h5", "EWG_Germany_ret_20d", "3M_ret_5d", "IWM_SmallCap_vol_20d", "HD_ret_20d", "TED_Spread_vol_20d", "LMT_LockheedMartin_ret_1d", "EWA_Australia_zscore_60d", "US3Y_Rate_ret_5d", "NVDA_vol_20d"], "is_new": true}, {"model_id": "new_h5_NORMAL_XGBoost_N20_t2", "algo": "XGBoost", "regime": "NORMAL", "horizon": 5, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "SO_SouthernCo_ret_5d", "US30Y_Rate_ret_20d", "3M_vol_20d", "INTC_ret_1d", "spx_abs_ret_max_5d", "Retail_Sales_zscore_60d", "FedFunds_zscore_60d", "US7Y_Rate_ret_20d", "T10Y2Y_Spread_ret_5d", "SBUX_vol_20d", "LOW_Lowes_ret_5d", "TED_Spread_zscore_60d", "EWY_Korea_zscore_60d", "BA_ret_1d", "EOG_EOGResources_ret_5d", "HangSeng_HK_ret_1d", "VRP_ma5", "EWS_Singapore_ret_5d"], "is_new": true}, {"model_id": "new_h5_NORMAL_XGBoost_N20_t3", "algo": "XGBoost", "regime": "NORMAL", "horizon": 5, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "TM_Telephone_ret_1d", "WTI_Oil_FRED_zscore_60d", "DHR_vol_20d", "PFE_ret_1d", "US3M_Rate_zscore_60d", "EWM_Malaysia_vol_20d", "Industrial_Production_zscore_60d", "vix_mean_abs_ret_5d", "HD_ret_20d", "FedFunds_zscore_60d", "GD_GeneralDynamics_zscore_60d", "TM_Telephone_vol_20d", "XLY_Disc_vol_20d", "AMT_AmericanTower_ret_1d", "JNJ_ret_1d", "ORCL_zscore_60d", "EWG_Germany_ret_20d", "hmm_p_stress"], "is_new": true}, {"model_id": "new_h5_NORMAL_XGBoost_N20_t4", "algo": "XGBoost", "regime": "NORMAL", "horizon": 5, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "MS_MorganStanley_zscore_60d", "Brent_Oil_FRED_ret_5d", "GILD_Gilead_ret_20d", "LMT_LockheedMartin_vol_20d", "PAYX_Paychex_ret_20d", "heston_var_ev_h3", "SPY_zscore_60d", "EOG_EOGResources_ret_5d", "HUM_Humana_ret_5d", "MO_AltriaMG_ret_1d", "M_Macys_vol_20d", "NWL_Newell_ret_20d", "EWM_Malaysia_vol_20d", "ORCL_vol_20d", "EWJ_Japan_vol_20d", "AXP_Amex_vol_20d", "LOW_Lowes_ret_20d", "CTAS_Cintas_vol_20d"], "is_new": true}, {"model_id": "new_h5_NORMAL_XGBoost_N20_t5", "algo": "XGBoost", "regime": "NORMAL", "horizon": 5, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "ORCL_zscore_60d", "XLY_Disc_vol_20d", "US7Y_Rate_ret_20d", "ORCL_vol_20d", "EWM_Malaysia_vol_20d", "CLX_Clorox_vol_20d", "PFE_ret_1d", "EWS_Singapore_ret_5d", "EWC_Canada_zscore_60d", "US30Y_Rate_ret_20d", "LMT_LockheedMartin_vol_20d", "MS_MorganStanley_zscore_60d", "US1Y_Rate_ret_20d", "gjr_condvar_h1", "EWY_Korea_ret_20d", "MS_MorganStanley_ret_5d", "US3Y_Rate_ret_5d", "heston_ev_h3"], "is_new": true}, {"model_id": "new_h5_NORMAL_XGBoost_N20_t6", "algo": "XGBoost", "regime": "NORMAL", "horizon": 5, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "Core_CPI_zscore_60d", "IYM_BasicMaterials_ret_20d", "ASX_Australia_ret_5d", "EWS_Singapore_ret_5d", "GE_ret_1d", "EWG_Germany_vol_20d", "Core_PCE_zscore_60d", "Nikkei_Japan_vol_20d", "EFFR_ret_1d", "XLK_Tech_zscore_60d", "EQIX_Equinix_ret_5d", "EWG_Germany_ret_20d", "ITT_ITTInc_ret_5d", "Michigan_Sentiment_ret_20d", "ES_Evergy_ret_1d", "DAX_Germany_vol_20d", "EWQ_France_zscore_60d", "FedFunds_zscore_60d"], "is_new": true}, {"model_id": "new_h5_NORMAL_XGBoost_N20_t7", "algo": "XGBoost", "regime": "NORMAL", "horizon": 5, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "CLX_Clorox_vol_20d", "EOG_EOGResources_vol_20d", "AMGN_Amgen_ret_1d", "ITT_ITTInc_ret_5d", "SBUX_vol_20d", "IBEX_Spain_ret_20d", "XLV_Health_zscore_60d", "DAX_Germany_zscore_60d", "BTI_BritishAmerican_ret_5d", "EWL_Switzerland_zscore_60d", "heston_var_ev_h3", "TGT_Target_zscore_60d", "NWL_Newell_ret_20d", "3M_ret_5d", "MSTR_Bitcoin3_ret_20d", "SLB_Schlumberger_ret_1d", "ES_Evergy_ret_1d", "TED_Spread_zscore_60d"], "is_new": true}, {"model_id": "new_h5_NORMAL_XGBoost_N25_t0", "algo": "XGBoost", "regime": "NORMAL", "horizon": 5, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "ASX_Australia_ret_5d", "DHR_vol_20d", "EWY_Korea_zscore_60d", "MO_AltriaMG_ret_1d", "M_Macys_vol_20d", "IBEX_Spain_ret_20d", "IWM_SmallCap_vol_20d", "EWM_Malaysia_zscore_60d", "Core_PCE_zscore_60d", "AXP_Amex_vol_20d", "PPL_PPL_ret_1d", "MS_MorganStanley_zscore_60d", "US7Y_Rate_ret_20d", "BDX_Becton_Dickinson_ret_20d", "Retail_Sales_zscore_60d", "AORD_AUS_zscore_60d", "AMD_ret_5d", "US5Y_Rate_ret_5d", "MSTR_Bitcoin3_ret_5d", "AVB_AvalonBay_zscore_60d", "T_ret_1d", "US3Y_Rate_ret_5d", "ORCL_vol_20d"], "is_new": true}, {"model_id": "new_h5_NORMAL_XGBoost_N25_t1", "algo": "XGBoost", "regime": "NORMAL", "horizon": 5, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "heston_var_ev_h5", "ITT_ITTInc_ret_5d", "VOD_Vodafone_zscore_60d", "EWM_Malaysia_ret_1d", "HD_ret_1d", "LUV_SouthwestAir_ret_5d", "SCHW_Schwab_ret_5d", "Brent_Oil_FRED_ret_20d", "NEE_NextEra_ret_20d", "CTAS_Cintas_vol_20d", "LOW_Lowes_ret_5d", "NWL_Newell_ret_20d", "EWG_Germany_vol_20d", "EWS_Singapore_ret_5d", "MS_MorganStanley_ret_1d", "AMGN_Amgen_ret_1d", "heston_var_ev_h3", "CCI_CrownCastle_vol_20d", "XLK_Tech_zscore_60d", "XLB_Materials_zscore_60d", "XLY_Disc_vol_20d", "EMR_Emerson_ret_20d", "EWJ_Japan_vol_20d"], "is_new": true}, {"model_id": "new_h5_NORMAL_XGBoost_N25_t2", "algo": "XGBoost", "regime": "NORMAL", "horizon": 5, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWM_Malaysia_vol_20d", "M_Macys_vol_20d", "SJM_JM_Smucker_ret_1d", "EFFR_vol_20d", "Nikkei_Japan_vol_20d", "SJM_JM_Smucker_ret_5d", "HD_ret_20d", "ORCL_zscore_60d", "vix_mean_abs_ret_5d", "AVB_AvalonBay_zscore_60d", "TM_Telephone_vol_20d", "CLX_Clorox_vol_20d", "NFCI_ret_5d", "VVIX_ret_20d", "heston_ev_h3", "EWC_Canada_zscore_60d", "JNJ_ret_1d", "AORD_AUS_zscore_60d", "EWH_HongKong_ret_5d", "GE_ret_1d", "DAX_Germany_zscore_60d", "AXP_Amex_ret_20d", "vix_acceleration_1d"], "is_new": true}, {"model_id": "new_h5_NORMAL_XGBoost_N25_t3", "algo": "XGBoost", "regime": "NORMAL", "horizon": 5, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "DE_Deere_vol_20d", "HD_zscore_60d", "US1Y_Rate_ret_20d", "IYM_BasicMaterials_ret_20d", "CPB_CampbellSoup_ret_5d", "XOM_ret_20d", "Brent_Oil_FRED_ret_5d", "Nikkei_Japan_zscore_60d", "TED_Spread_vol_20d", "US1Y_Rate_ret_5d", "EWM_Malaysia_ret_1d", "US7Y_Rate_ret_20d", "EFFR_vol_20d", "Brent_Oil_FRED_ret_20d", "BTI_BritishAmerican_ret_20d", "IBEX_Spain_ret_20d", "EWY_Korea_ret_20d", "MSTR_Bitcoin3_ret_1d", "SCHW_Schwab_ret_5d", "T_ret_1d", "LLY_zscore_60d", "TED_Spread_zscore_60d", "PAYX_Paychex_zscore_60d"], "is_new": true}, {"model_id": "new_h5_NORMAL_XGBoost_N25_t4", "algo": "XGBoost", "regime": "NORMAL", "horizon": 5, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "MSTR_Bitcoin3_ret_1d", "SBUX_ret_5d", "Nikkei_Japan_vol_20d", "M_Macys_vol_20d", "TED_Spread_zscore_60d", "DAX_Germany_zscore_60d", "XLV_Health_zscore_60d", "EWH_HongKong_ret_5d", "BA_ret_1d", "AMZN_ret_5d", "SBUX_vol_20d", "heston_var_ev_h5", "PPL_PPL_ret_1d", "AMD_ret_5d", "EWQ_France_zscore_60d", "SLB_Schlumberger_ret_1d", "PAYX_Paychex_zscore_60d", "SCHW_Schwab_ret_5d", "HD_zscore_60d", "Nikkei_Japan_zscore_60d", "GD_GeneralDynamics_zscore_60d", "EWM_Malaysia_zscore_60d", "XLK_Tech_zscore_60d"], "is_new": true}, {"model_id": "new_h5_NORMAL_XGBoost_N25_t5", "algo": "XGBoost", "regime": "NORMAL", "horizon": 5, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "MRK_Merck_zscore_60d", "PAYX_Paychex_zscore_60d", "AORD_AUS_zscore_60d", "HD_ret_20d", "GD_GeneralDynamics_zscore_60d", "Industrial_Production_zscore_60d", "LMT_LockheedMartin_ret_1d", "EWY_Korea_zscore_60d", "EMR_Emerson_ret_20d", "US1Y_Rate_ret_5d", "EWS_Singapore_ret_5d", "MO_AltriaMG_ret_1d", "SO_SouthernCo_ret_5d", "TM_Telephone_vol_20d", "Retail_Sales_zscore_60d", "EXC_Exelon_ret_1d", "EOG_EOGResources_vol_20d", "CPB_CampbellSoup_ret_20d", "AXP_Amex_vol_20d", "US6M_Rate_ret_20d", "spx_abs_ret_max_5d", "EWM_Malaysia_zscore_60d", "TXN_vol_20d"], "is_new": true}, {"model_id": "new_h5_NORMAL_XGBoost_N25_t6", "algo": "XGBoost", "regime": "NORMAL", "horizon": 5, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "VRP_ma5", "hmm_p_stress", "FedFunds_zscore_60d", "SLB_Schlumberger_ret_5d", "vix_mean_abs_ret_5d", "spx_momentum_3d", "EWM_Malaysia_zscore_60d", "3M_vol_20d", "Core_PCE_zscore_60d", "T10Y2Y_Spread_ret_5d", "NWL_Newell_ret_20d", "AXP_Amex_vol_20d", "TM_Telephone_ret_1d", "AORD_AUS_zscore_60d", "PLD_Prologis_ret_5d", "MO_AltriaMG_ret_1d", "EWL_Switzerland_zscore_60d", "US3Y_Rate_ret_5d", "EWM_Malaysia_vol_20d", "spx_abs_ret_max_5d", "US1Y_Rate_ret_20d", "LOW_Lowes_ret_5d", "MSTR_Bitcoin3_ret_1d"], "is_new": true}, {"model_id": "new_h5_NORMAL_XGBoost_N25_t7", "algo": "XGBoost", "regime": "NORMAL", "horizon": 5, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWY_Korea_zscore_60d", "EQR_Equity_ret_1d", "Retail_Sales_zscore_60d", "AVB_AvalonBay_zscore_60d", "VRP_ma5", "SO_SouthernCo_ret_5d", "EWQ_France_zscore_60d", "XLB_Materials_zscore_60d", "NFCI_ret_5d", "IYR_US_REIT2_zscore_60d", "HD_zscore_60d", "TED_Spread_vol_20d", "US7Y_Rate_ret_20d", "HangSeng_HK_ret_1d", "EWC_Canada_zscore_60d", "EWL_Switzerland_vol_20d", "hmm_p_stress", "AMD_ret_1d", "LUV_SouthwestAir_ret_5d", "XLK_Tech_zscore_60d", "heston_var_ev_h3", "EWM_Malaysia_zscore_60d", "vix_mean_abs_ret_5d"], "is_new": true}, {"model_id": "new_h5_NORMAL_XGBoost_N30_t0", "algo": "XGBoost", "regime": "NORMAL", "horizon": 5, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "SJM_JM_Smucker_ret_5d", "AVB_AvalonBay_zscore_60d", "EXC_Exelon_ret_1d", "PAYX_Paychex_ret_20d", "hmm_p_stress", "US1Y_Rate_ret_20d", "BLK_BlackRock_zscore_60d", "LLY_zscore_60d", "gjr_condvar_h1", "IBEX_Spain_ret_20d", "EWY_Korea_zscore_60d", "LMT_LockheedMartin_ret_1d", "DE_Deere_ret_5d", "US7Y_Rate_ret_20d", "EOG_EOGResources_ret_5d", "MO_AltriaMG_ret_1d", "SO_SouthernCo_ret_5d", "ES_Evergy_ret_1d", "FedFunds_zscore_60d", "US6M_Rate_ret_20d", "CPB_CampbellSoup_zscore_60d", "HD_ret_5d", "TXN_vol_20d", "3M_vol_20d", "US1Y_Rate_ret_5d", "PAYX_Paychex_vol_20d", "EXC_Exelon_zscore_60d", "IWM_SmallCap_vol_20d"], "is_new": true}, {"model_id": "new_h5_NORMAL_XGBoost_N30_t1", "algo": "XGBoost", "regime": "NORMAL", "horizon": 5, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "CPB_CampbellSoup_ret_5d", "AVB_AvalonBay_zscore_60d", "LLY_zscore_60d", "T10Y2Y_Spread_ret_5d", "SBUX_ret_5d", "MS_MorganStanley_ret_1d", "spx_momentum_3d", "M_Macys_vol_20d", "WTI_Oil_FRED_zscore_60d", "EXC_Exelon_ret_1d", "MSTR_Bitcoin3_ret_20d", "BTI_BritishAmerican_ret_20d", "T_ret_1d", "HD_ret_1d", "CI_Cigna_vol_20d", "MSTR_Bitcoin3_ret_5d", "ITT_ITTInc_ret_5d", "SJM_JM_Smucker_ret_1d", "XOM_ret_20d", "EXC_Exelon_zscore_60d", "CPB_CampbellSoup_vol_20d", "XLB_Materials_zscore_60d", "US1Y_Rate_ret_20d", "PAYX_Paychex_zscore_60d", "IBEX_Spain_ret_20d", "Core_PCE_zscore_60d", "AORD_AUS_zscore_60d", "IWM_SmallCap_vol_20d"], "is_new": true}, {"model_id": "new_h5_NORMAL_XGBoost_N30_t2", "algo": "XGBoost", "regime": "NORMAL", "horizon": 5, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "NFCI_ret_5d", "EQIX_Equinix_ret_5d", "GILD_Gilead_ret_20d", "NWL_Newell_ret_20d", "EWL_Switzerland_vol_20d", "ASX_Australia_ret_5d", "EQR_Equity_ret_1d", "JNJ_ret_1d", "heston_var_ev_h3", "SCHW_Schwab_ret_5d", "TM_Telephone_vol_20d", "NOC_Northrop_ret_20d", "Industrial_Production_zscore_60d", "EWM_Malaysia_zscore_60d", "IYR_US_REIT2_zscore_60d", "SBUX_ret_5d", "US1Y_Rate_ret_20d", "MSTR_Bitcoin3_ret_5d", "NVDA_vol_20d", "MS_MorganStanley_ret_5d", "PFE_ret_1d", "EWJ_Japan_vol_20d", "Nikkei_Japan_vol_20d", "AORD_AUS_zscore_60d", "EWL_Switzerland_zscore_60d", "DIS_vol_20d", "CPB_CampbellSoup_ret_20d", "CTAS_Cintas_vol_20d"], "is_new": true}, {"model_id": "new_h5_NORMAL_XGBoost_N30_t3", "algo": "XGBoost", "regime": "NORMAL", "horizon": 5, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "US30Y_Rate_ret_20d", "Nikkei_Japan_vol_20d", "CI_Cigna_vol_20d", "LOW_Lowes_ret_20d", "GILD_Gilead_ret_20d", "SBUX_ret_5d", "XLK_Tech_zscore_60d", "US5Y_Rate_ret_5d", "SLB_Schlumberger_ret_1d", "HangSeng_HK_ret_1d", "FedFunds_zscore_60d", "CPB_CampbellSoup_ret_5d", "TXN_vol_20d", "LMT_LockheedMartin_ret_1d", "BTI_BritishAmerican_ret_20d", "WTI_Oil_FRED_zscore_60d", "VVIX_ret_20d", "NFCI_ret_5d", "SO_SouthernCo_ret_5d", "EWG_Germany_vol_20d", "EFFR_vol_20d", "hmm_p_stress", "NVDA_vol_20d", "DHR_vol_20d", "TGT_Target_zscore_60d", "BDX_Becton_Dickinson_ret_20d", "SLB_Schlumberger_ret_5d", "US6M_Rate_ret_20d"], "is_new": true}, {"model_id": "new_h5_NORMAL_XGBoost_N30_t4", "algo": "XGBoost", "regime": "NORMAL", "horizon": 5, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "MRK_Merck_zscore_60d", "VRP_ma5", "NVDA_vol_20d", "AMD_ret_1d", "ENB_EnbridgeInc_ret_1d", "EWC_Canada_zscore_60d", "EWM_Malaysia_ret_1d", "XLF_Fin_vol_20d", "SLB_Schlumberger_ret_5d", "AMT_AmericanTower_ret_1d", "GILD_Gilead_ret_20d", "PAYX_Paychex_vol_20d", "EWS_Singapore_ret_5d", "DOW_Price_zscore_60d", "SBUX_zscore_60d", "EQR_Equity_ret_1d", "DAX_Germany_zscore_60d", "GE_ret_1d", "EWA_Australia_ret_1d", "XLV_Health_zscore_60d", "INTC_ret_1d", "TM_Telephone_vol_20d", "CPB_CampbellSoup_ret_5d", "CPB_CampbellSoup_vol_20d", "Nikkei_Japan_vol_20d", "DE_Deere_vol_20d", "LLY_zscore_60d", "MS_MorganStanley_ret_1d"], "is_new": true}, {"model_id": "new_h5_NORMAL_XGBoost_N30_t5", "algo": "XGBoost", "regime": "NORMAL", "horizon": 5, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "Brent_Oil_FRED_ret_20d", "Core_PCE_zscore_60d", "GD_GeneralDynamics_zscore_60d", "PAYX_Paychex_ret_20d", "spx_momentum_3d", "PLD_Prologis_ret_5d", "vix_acceleration_1d", "MSTR_Bitcoin3_ret_5d", "EWG_Germany_ret_20d", "Core_CPI_zscore_60d", "INTC_ret_1d", "XLV_Health_zscore_60d", "EWC_Canada_zscore_60d", "TED_Spread_vol_20d", "AXP_Amex_vol_20d", "VVIX_ret_20d", "M_Macys_vol_20d", "DHR_vol_20d", "PG_ret_20d", "SLB_Schlumberger_ret_5d", "MS_MorganStanley_ret_5d", "CTAS_Cintas_vol_20d", "CMCSA_ret_1d", "SPY_zscore_60d", "US3Y_Rate_ret_5d", "BTI_BritishAmerican_ret_20d", "US3M_Rate_vol_20d", "ENB_EnbridgeInc_ret_1d"], "is_new": true}, {"model_id": "new_h5_NORMAL_XGBoost_N30_t6", "algo": "XGBoost", "regime": "NORMAL", "horizon": 5, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "ASX_Australia_vol_20d", "DE_Deere_ret_5d", "ORCL_vol_20d", "heston_var_ev_h5", "AXP_Amex_ret_20d", "SO_SouthernCo_ret_5d", "EWM_Malaysia_ret_1d", "IYR_US_REIT2_zscore_60d", "US1Y_Rate_ret_5d", "TM_Telephone_vol_20d", "US3M_Rate_zscore_60d", "PG_ret_20d", "HD_ret_20d", "EWJ_Japan_vol_20d", "MRK_Merck_zscore_60d", "HangSeng_HK_vol_20d", "BLK_BlackRock_zscore_60d", "SLB_Schlumberger_ret_5d", "INTC_ret_1d", "T_ret_1d", "HD_zscore_60d", "EFFR_vol_20d", "3M_ret_5d", "HangSeng_HK_ret_1d", "LOW_Lowes_ret_5d", "EWY_Korea_zscore_60d", "EQIX_Equinix_ret_5d", "SJM_JM_Smucker_ret_5d"], "is_new": true}, {"model_id": "new_h5_NORMAL_XGBoost_N30_t7", "algo": "XGBoost", "regime": "NORMAL", "horizon": 5, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "Nikkei_Japan_zscore_60d", "PAYX_Paychex_zscore_60d", "INTC_ret_1d", "EWM_Malaysia_vol_20d", "T10Y2Y_Spread_ret_5d", "NEE_NextEra_ret_20d", "hmm_p_stress", "US30Y_Rate_ret_20d", "SPY_zscore_60d", "Brent_Oil_FRED_ret_5d", "IBEX_Spain_ret_20d", "JNJ_ret_1d", "EWJ_Japan_vol_20d", "VRP_ma5", "DE_Deere_ret_5d", "HangSeng_HK_vol_20d", "CLX_Clorox_vol_20d", "EWG_Germany_ret_20d", "XLB_Materials_zscore_60d", "SBUX_zscore_60d", "EWG_Germany_vol_20d", "EWA_Australia_ret_1d", "EOG_EOGResources_vol_20d", "MS_MorganStanley_zscore_60d", "HangSeng_HK_ret_5d", "BA_ret_1d", "spx_abs_ret_max_5d", "CPB_CampbellSoup_ret_20d"], "is_new": true}, {"model_id": "new_h5_NORMAL_LightGBM_N5_t0", "algo": "LightGBM", "regime": "NORMAL", "horizon": 5, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "US30Y_Rate_ret_20d", "US3Y_Rate_ret_5d", "BLK_BlackRock_zscore_60d"], "is_new": true}, {"model_id": "new_h5_NORMAL_LightGBM_N5_t1", "algo": "LightGBM", "regime": "NORMAL", "horizon": 5, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "PLD_Prologis_ret_5d", "EWM_Malaysia_zscore_60d", "LOW_Lowes_ret_20d"], "is_new": true}, {"model_id": "new_h5_NORMAL_LightGBM_N5_t2", "algo": "LightGBM", "regime": "NORMAL", "horizon": 5, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "INTC_ret_1d", "QQQ_vol_20d", "vix_mean_abs_ret_5d"], "is_new": true}, {"model_id": "new_h5_NORMAL_LightGBM_N5_t3", "algo": "LightGBM", "regime": "NORMAL", "horizon": 5, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "TED_Spread_zscore_60d", "EMR_Emerson_ret_20d", "US3M_Rate_vol_20d"], "is_new": true}, {"model_id": "new_h5_NORMAL_LightGBM_N5_t4", "algo": "LightGBM", "regime": "NORMAL", "horizon": 5, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "INTC_ret_1d", "DIS_vol_20d", "SLB_Schlumberger_ret_5d"], "is_new": true}, {"model_id": "new_h5_NORMAL_LightGBM_N5_t5", "algo": "LightGBM", "regime": "NORMAL", "horizon": 5, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "US5Y_Rate_ret_5d", "AXP_Amex_vol_20d", "MSTR_Bitcoin3_ret_1d"], "is_new": true}, {"model_id": "new_h5_NORMAL_LightGBM_N5_t6", "algo": "LightGBM", "regime": "NORMAL", "horizon": 5, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "DOW_Price_zscore_60d", "BTI_BritishAmerican_ret_20d", "XOM_ret_20d"], "is_new": true}, {"model_id": "new_h5_NORMAL_LightGBM_N5_t7", "algo": "LightGBM", "regime": "NORMAL", "horizon": 5, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "Michigan_Sentiment_ret_20d", "EFFR_ret_1d", "EWY_Korea_zscore_60d"], "is_new": true}, {"model_id": "new_h5_NORMAL_LightGBM_N8_t0", "algo": "LightGBM", "regime": "NORMAL", "horizon": 5, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EFFR_ret_1d", "XLK_Tech_zscore_60d", "ASX_Australia_vol_20d", "NWL_Newell_ret_20d", "NVDA_vol_20d", "LMT_LockheedMartin_ret_1d"], "is_new": true}, {"model_id": "new_h5_NORMAL_LightGBM_N8_t1", "algo": "LightGBM", "regime": "NORMAL", "horizon": 5, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "SCHW_Schwab_ret_5d", "BA_ret_1d", "EWA_Australia_ret_1d", "ORCL_zscore_60d", "AXP_Amex_vol_20d", "BTI_BritishAmerican_ret_5d"], "is_new": true}, {"model_id": "new_h5_NORMAL_LightGBM_N8_t2", "algo": "LightGBM", "regime": "NORMAL", "horizon": 5, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "LMT_LockheedMartin_vol_20d", "EWM_Malaysia_zscore_60d", "SLB_Schlumberger_ret_1d", "TED_Spread_zscore_60d", "DE_Deere_ret_5d", "JNJ_ret_1d"], "is_new": true}, {"model_id": "new_h5_NORMAL_LightGBM_N8_t3", "algo": "LightGBM", "regime": "NORMAL", "horizon": 5, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "PFE_ret_1d", "CMCSA_ret_1d", "CPB_CampbellSoup_vol_20d", "QQQ_vol_20d", "heston_var_ev_h7", "Brent_Oil_FRED_ret_20d"], "is_new": true}, {"model_id": "new_h5_NORMAL_LightGBM_N8_t4", "algo": "LightGBM", "regime": "NORMAL", "horizon": 5, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "BTI_BritishAmerican_ret_20d", "PPL_PPL_ret_1d", "TXN_vol_20d", "CCI_CrownCastle_vol_20d", "SJM_JM_Smucker_ret_5d", "US30Y_Rate_ret_20d"], "is_new": true}, {"model_id": "new_h5_NORMAL_LightGBM_N8_t5", "algo": "LightGBM", "regime": "NORMAL", "horizon": 5, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "T_ret_1d", "VOD_Vodafone_zscore_60d", "Core_PCE_zscore_60d", "TED_Spread_zscore_60d", "gjr_condvar_h1", "Nikkei_Japan_vol_20d"], "is_new": true}, {"model_id": "new_h5_NORMAL_LightGBM_N8_t6", "algo": "LightGBM", "regime": "NORMAL", "horizon": 5, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "SPY_zscore_60d", "GD_GeneralDynamics_zscore_60d", "BDX_Becton_Dickinson_ret_20d", "SJM_JM_Smucker_ret_5d", "heston_var_ev_h7", "PG_ret_20d"], "is_new": true}, {"model_id": "new_h5_NORMAL_LightGBM_N8_t7", "algo": "LightGBM", "regime": "NORMAL", "horizon": 5, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "LLY_zscore_60d", "M_Macys_vol_20d", "EQIX_Equinix_ret_5d", "CPB_CampbellSoup_zscore_60d", "AMD_ret_1d", "AXP_Amex_vol_20d"], "is_new": true}, {"model_id": "new_h5_NORMAL_LightGBM_N10_t0", "algo": "LightGBM", "regime": "NORMAL", "horizon": 5, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "US3Y_Rate_ret_5d", "HD_zscore_60d", "AXP_Amex_ret_20d", "ORCL_vol_20d", "spx_abs_ret_max_5d", "IYM_BasicMaterials_ret_20d", "Nikkei_Japan_zscore_60d", "SPY_zscore_60d"], "is_new": true}, {"model_id": "new_h5_NORMAL_LightGBM_N10_t1", "algo": "LightGBM", "regime": "NORMAL", "horizon": 5, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "PAYX_Paychex_ret_20d", "3M_vol_20d", "TM_Telephone_vol_20d", "NOC_Northrop_ret_20d", "BTI_BritishAmerican_ret_20d", "HangSeng_HK_ret_5d", "NWL_Newell_ret_20d", "EWG_Germany_vol_20d"], "is_new": true}, {"model_id": "new_h5_NORMAL_LightGBM_N10_t2", "algo": "LightGBM", "regime": "NORMAL", "horizon": 5, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "PFE_ret_1d", "heston_ev_h3", "HUM_Humana_ret_5d", "EOG_EOGResources_vol_20d", "EWL_Switzerland_vol_20d", "PPL_PPL_ret_1d", "SLB_Schlumberger_ret_1d", "EWA_Australia_zscore_60d"], "is_new": true}, {"model_id": "new_h5_NORMAL_LightGBM_N10_t3", "algo": "LightGBM", "regime": "NORMAL", "horizon": 5, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "CTAS_Cintas_vol_20d", "GILD_Gilead_ret_20d", "PPL_PPL_ret_1d", "3M_vol_20d", "CMCSA_ret_1d", "DIS_vol_20d", "MS_MorganStanley_ret_5d", "AMGN_Amgen_ret_1d"], "is_new": true}, {"model_id": "new_h5_NORMAL_LightGBM_N10_t4", "algo": "LightGBM", "regime": "NORMAL", "horizon": 5, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "XOM_ret_1d", "DAX_Germany_zscore_60d", "Michigan_Sentiment_ret_20d", "MS_MorganStanley_ret_1d", "VOD_Vodafone_zscore_60d", "EMR_Emerson_ret_20d", "EWM_Malaysia_zscore_60d", "DE_Deere_vol_20d"], "is_new": true}, {"model_id": "new_h5_NORMAL_LightGBM_N10_t5", "algo": "LightGBM", "regime": "NORMAL", "horizon": 5, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "DHR_vol_20d", "CPB_CampbellSoup_vol_20d", "EWY_Korea_zscore_60d", "SLB_Schlumberger_ret_5d", "SJM_JM_Smucker_ret_1d", "ORCL_vol_20d", "EWM_Malaysia_vol_20d", "AORD_AUS_zscore_60d"], "is_new": true}, {"model_id": "new_h5_NORMAL_LightGBM_N10_t6", "algo": "LightGBM", "regime": "NORMAL", "horizon": 5, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "CCI_CrownCastle_vol_20d", "US1Y_Rate_ret_20d", "AMGN_Amgen_ret_1d", "EWG_Germany_ret_20d", "Core_PCE_zscore_60d", "VOD_Vodafone_zscore_60d", "MS_MorganStanley_ret_5d", "T10Y2Y_Spread_ret_5d"], "is_new": true}, {"model_id": "new_h5_NORMAL_LightGBM_N10_t7", "algo": "LightGBM", "regime": "NORMAL", "horizon": 5, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "VRP_ma5", "BTI_BritishAmerican_ret_5d", "EWA_Australia_zscore_60d", "LMT_LockheedMartin_ret_1d", "SLB_Schlumberger_ret_1d", "ES_Evergy_ret_1d", "Nikkei_Japan_vol_20d", "EWY_Korea_ret_20d"], "is_new": true}, {"model_id": "new_h5_NORMAL_LightGBM_N12_t0", "algo": "LightGBM", "regime": "NORMAL", "horizon": 5, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "HangSeng_HK_ret_1d", "IYM_BasicMaterials_ret_20d", "XLY_Disc_vol_20d", "DAX_Germany_vol_20d", "XLF_Fin_vol_20d", "T10Y2Y_Spread_ret_5d", "SJM_JM_Smucker_ret_1d", "TGT_Target_zscore_60d", "CTAS_Cintas_vol_20d", "EWL_Switzerland_vol_20d"], "is_new": true}, {"model_id": "new_h5_NORMAL_LightGBM_N12_t1", "algo": "LightGBM", "regime": "NORMAL", "horizon": 5, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWC_Canada_zscore_60d", "DE_Deere_ret_5d", "US1Y_Rate_ret_20d", "SPY_zscore_60d", "EFFR_vol_20d", "heston_var_ev_h5", "EWM_Malaysia_ret_1d", "XLY_Disc_vol_20d", "EQR_Equity_ret_1d", "PFE_ret_1d"], "is_new": true}, {"model_id": "new_h5_NORMAL_LightGBM_N12_t2", "algo": "LightGBM", "regime": "NORMAL", "horizon": 5, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "TED_Spread_zscore_60d", "PPL_PPL_ret_1d", "AXP_Amex_vol_20d", "MRK_Merck_zscore_60d", "INTC_ret_5d", "HD_ret_5d", "HD_ret_20d", "AMZN_ret_5d", "IYR_US_REIT2_zscore_60d", "HangSeng_HK_ret_1d"], "is_new": true}, {"model_id": "new_h5_NORMAL_LightGBM_N12_t3", "algo": "LightGBM", "regime": "NORMAL", "horizon": 5, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EQR_Equity_ret_1d", "HUM_Humana_ret_5d", "CTAS_Cintas_vol_20d", "EWG_Germany_ret_20d", "NVDA_vol_20d", "XLY_Disc_vol_20d", "AVB_AvalonBay_zscore_60d", "Retail_Sales_zscore_60d", "ASX_Australia_ret_5d", "EFFR_vol_20d"], "is_new": true}, {"model_id": "new_h5_NORMAL_LightGBM_N12_t4", "algo": "LightGBM", "regime": "NORMAL", "horizon": 5, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "CPB_CampbellSoup_vol_20d", "AMD_ret_5d", "PPL_PPL_ret_1d", "CPB_CampbellSoup_ret_5d", "US1Y_Rate_ret_20d", "PFE_ret_1d", "SO_SouthernCo_ret_5d", "US3M_Rate_vol_20d", "NVDA_vol_20d", "SCHW_Schwab_ret_5d"], "is_new": true}, {"model_id": "new_h5_NORMAL_LightGBM_N12_t5", "algo": "LightGBM", "regime": "NORMAL", "horizon": 5, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "3M_ret_5d", "GILD_Gilead_ret_20d", "INTC_ret_1d", "HangSeng_HK_ret_5d", "JNJ_ret_1d", "SJM_JM_Smucker_ret_1d", "XLF_Fin_vol_20d", "HD_ret_20d", "DHR_ret_1d", "MO_AltriaMG_ret_1d"], "is_new": true}, {"model_id": "new_h5_NORMAL_LightGBM_N12_t6", "algo": "LightGBM", "regime": "NORMAL", "horizon": 5, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWH_HongKong_ret_5d", "ENB_EnbridgeInc_ret_1d", "MS_MorganStanley_zscore_60d", "US5Y_Rate_ret_5d", "LLY_zscore_60d", "EWG_Germany_ret_20d", "PAYX_Paychex_zscore_60d", "DIS_vol_20d", "EXC_Exelon_ret_1d", "IBEX_Spain_ret_20d"], "is_new": true}, {"model_id": "new_h5_NORMAL_LightGBM_N12_t7", "algo": "LightGBM", "regime": "NORMAL", "horizon": 5, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EFFR_vol_20d", "MSTR_Bitcoin3_ret_1d", "VRP_ma5", "INTC_ret_1d", "Brent_Oil_FRED_ret_20d", "HD_zscore_60d", "hmm_p_stress", "heston_var_ev_h3", "EWQ_France_zscore_60d", "NOC_Northrop_ret_20d"], "is_new": true}, {"model_id": "new_h5_NORMAL_LightGBM_N15_t0", "algo": "LightGBM", "regime": "NORMAL", "horizon": 5, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "US6M_Rate_ret_20d", "CPB_CampbellSoup_ret_20d", "DHR_vol_20d", "LLY_zscore_60d", "SLB_Schlumberger_ret_5d", "MRK_Merck_zscore_60d", "EWQ_France_ret_20d", "IBEX_Spain_ret_20d", "BA_ret_1d", "DE_Deere_vol_20d", "WTI_Oil_FRED_zscore_60d", "SBUX_vol_20d", "AMGN_Amgen_ret_1d"], "is_new": true}, {"model_id": "new_h5_NORMAL_LightGBM_N15_t1", "algo": "LightGBM", "regime": "NORMAL", "horizon": 5, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "AMT_AmericanTower_ret_1d", "ORCL_vol_20d", "spx_vol_5d", "EXC_Exelon_zscore_60d", "ASX_Australia_ret_5d", "Brent_Oil_FRED_ret_5d", "EWG_Germany_vol_20d", "SCHW_Schwab_ret_5d", "GD_GeneralDynamics_zscore_60d", "US1Y_Rate_ret_20d", "US5Y_Rate_ret_5d", "MS_MorganStanley_ret_5d", "SJM_JM_Smucker_ret_1d"], "is_new": true}, {"model_id": "new_h5_NORMAL_LightGBM_N15_t2", "algo": "LightGBM", "regime": "NORMAL", "horizon": 5, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "SBUX_zscore_60d", "MSTR_Bitcoin3_ret_20d", "XLF_Fin_vol_20d", "DAX_Germany_zscore_60d", "FedFunds_zscore_60d", "MS_MorganStanley_zscore_60d", "BTI_BritishAmerican_ret_5d", "3M_vol_20d", "IYM_BasicMaterials_ret_20d", "US3Y_Rate_ret_5d", "gjr_condvar_h1", "WTI_Oil_FRED_zscore_60d", "heston_var_ev_h7"], "is_new": true}, {"model_id": "new_h5_NORMAL_LightGBM_N15_t3", "algo": "LightGBM", "regime": "NORMAL", "horizon": 5, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "vix_acceleration_1d", "HangSeng_HK_ret_1d", "SBUX_vol_20d", "XLF_Fin_vol_20d", "HD_ret_20d", "AMZN_ret_5d", "IWM_SmallCap_vol_20d", "ASX_Australia_vol_20d", "CPB_CampbellSoup_vol_20d", "IYR_US_REIT2_zscore_60d", "US7Y_Rate_ret_20d", "GE_ret_1d", "XLB_Materials_zscore_60d"], "is_new": true}, {"model_id": "new_h5_NORMAL_LightGBM_N15_t4", "algo": "LightGBM", "regime": "NORMAL", "horizon": 5, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "IYM_BasicMaterials_ret_20d", "NVDA_vol_20d", "heston_ev_h3", "EMR_Emerson_ret_20d", "PG_ret_20d", "AMD_ret_1d", "MSTR_Bitcoin3_ret_5d", "Brent_Oil_FRED_ret_5d", "AMGN_Amgen_ret_1d", "spx_momentum_3d", "LOW_Lowes_ret_5d", "AVB_AvalonBay_zscore_60d", "BDX_Becton_Dickinson_ret_20d"], "is_new": true}, {"model_id": "new_h5_NORMAL_LightGBM_N15_t5", "algo": "LightGBM", "regime": "NORMAL", "horizon": 5, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "XLY_Disc_vol_20d", "XOM_ret_20d", "LUV_SouthwestAir_ret_5d", "EWS_Singapore_ret_5d", "ORCL_zscore_60d", "DOW_Price_zscore_60d", "US1Y_Rate_ret_5d", "CPB_CampbellSoup_ret_20d", "PG_ret_20d", "SBUX_ret_5d", "FedFunds_zscore_60d", "3M_vol_20d", "XLV_Health_zscore_60d"], "is_new": true}, {"model_id": "new_h5_NORMAL_LightGBM_N15_t6", "algo": "LightGBM", "regime": "NORMAL", "horizon": 5, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "LUV_SouthwestAir_ret_5d", "spx_momentum_3d", "EWJ_Japan_vol_20d", "Brent_Oil_FRED_ret_5d", "Core_PCE_zscore_60d", "LOW_Lowes_ret_5d", "HD_zscore_60d", "ITT_ITTInc_ret_5d", "BTI_BritishAmerican_ret_20d", "NVDA_vol_20d", "HangSeng_HK_ret_1d", "EWS_Singapore_ret_5d", "heston_var_ev_h5"], "is_new": true}, {"model_id": "new_h5_NORMAL_LightGBM_N15_t7", "algo": "LightGBM", "regime": "NORMAL", "horizon": 5, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "PG_ret_20d", "3M_ret_5d", "AVB_AvalonBay_zscore_60d", "VRP_ma5", "PCAR_PaccarInc_ret_5d", "EWY_Korea_zscore_60d", "CPB_CampbellSoup_vol_20d", "US1Y_Rate_ret_20d", "T_ret_1d", "NOC_Northrop_ret_20d", "DHR_ret_1d", "SBUX_zscore_60d", "LMT_LockheedMartin_vol_20d"], "is_new": true}, {"model_id": "new_h5_NORMAL_LightGBM_N20_t0", "algo": "LightGBM", "regime": "NORMAL", "horizon": 5, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "heston_ev_h3", "EWA_Australia_zscore_60d", "BA_ret_1d", "ORCL_zscore_60d", "JNJ_ret_1d", "LLY_zscore_60d", "QQQ_vol_20d", "T_ret_1d", "NEE_NextEra_ret_20d", "hmm_p_stress", "CPB_CampbellSoup_ret_20d", "MSTR_Bitcoin3_ret_20d", "INTC_ret_5d", "US1Y_Rate_ret_20d", "EWY_Korea_zscore_60d", "CPB_CampbellSoup_ret_5d", "EWC_Canada_zscore_60d", "3M_vol_20d"], "is_new": true}, {"model_id": "new_h5_NORMAL_LightGBM_N20_t1", "algo": "LightGBM", "regime": "NORMAL", "horizon": 5, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "AMT_AmericanTower_ret_1d", "NEE_NextEra_ret_20d", "HD_zscore_60d", "M_Macys_vol_20d", "EQIX_Equinix_ret_5d", "NVDA_vol_20d", "hmm_p_stress", "Nikkei_Japan_vol_20d", "SBUX_vol_20d", "EOG_EOGResources_vol_20d", "XLV_Health_zscore_60d", "SJM_JM_Smucker_ret_5d", "EWQ_France_zscore_60d", "NOC_Northrop_ret_20d", "CI_Cigna_vol_20d", "VRP_ma5", "XOM_ret_20d", "HangSeng_HK_vol_20d"], "is_new": true}, {"model_id": "new_h5_NORMAL_LightGBM_N20_t2", "algo": "LightGBM", "regime": "NORMAL", "horizon": 5, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "SJM_JM_Smucker_ret_5d", "vix_acceleration_1d", "EQIX_Equinix_ret_5d", "VVIX_ret_20d", "Michigan_Sentiment_ret_20d", "EWG_Germany_ret_20d", "SLB_Schlumberger_ret_5d", "3M_vol_20d", "ORCL_vol_20d", "IBEX_Spain_ret_20d", "spx_abs_ret_max_5d", "EFFR_ret_1d", "US30Y_Rate_ret_20d", "BA_ret_1d", "EWM_Malaysia_zscore_60d", "Brent_Oil_FRED_ret_5d", "LLY_zscore_60d", "CPB_CampbellSoup_zscore_60d"], "is_new": true}, {"model_id": "new_h5_NORMAL_LightGBM_N20_t3", "algo": "LightGBM", "regime": "NORMAL", "horizon": 5, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "INTC_ret_1d", "heston_var_ev_h5", "TED_Spread_vol_20d", "CI_Cigna_vol_20d", "PAYX_Paychex_ret_20d", "PG_ret_20d", "EOG_EOGResources_vol_20d", "US6M_Rate_ret_20d", "heston_var_ev_h7", "ES_Evergy_ret_1d", "LLY_zscore_60d", "PLD_Prologis_ret_5d", "TED_Spread_zscore_60d", "EWC_Canada_zscore_60d", "NWL_Newell_ret_20d", "EWM_Malaysia_zscore_60d", "NOC_Northrop_ret_20d", "CPB_CampbellSoup_zscore_60d"], "is_new": true}, {"model_id": "new_h5_NORMAL_LightGBM_N20_t4", "algo": "LightGBM", "regime": "NORMAL", "horizon": 5, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWL_Switzerland_zscore_60d", "DOW_Price_zscore_60d", "IWM_SmallCap_vol_20d", "DHR_ret_1d", "NWL_Newell_ret_20d", "AXP_Amex_vol_20d", "AVB_AvalonBay_zscore_60d", "BTI_BritishAmerican_ret_5d", "heston_ev_h3", "CLX_Clorox_vol_20d", "TM_Telephone_vol_20d", "heston_var_ev_h3", "CPB_CampbellSoup_zscore_60d", "ES_Evergy_ret_1d", "EWS_Singapore_ret_5d", "AMGN_Amgen_ret_1d", "Retail_Sales_zscore_60d", "EWJ_Japan_vol_20d"], "is_new": true}, {"model_id": "new_h5_NORMAL_LightGBM_N20_t5", "algo": "LightGBM", "regime": "NORMAL", "horizon": 5, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "ENB_EnbridgeInc_ret_1d", "CLX_Clorox_vol_20d", "DE_Deere_vol_20d", "FedFunds_zscore_60d", "XOM_ret_20d", "EWC_Canada_zscore_60d", "NOC_Northrop_ret_20d", "LMT_LockheedMartin_ret_1d", "DAX_Germany_zscore_60d", "CPB_CampbellSoup_ret_20d", "BLK_BlackRock_zscore_60d", "Brent_Oil_FRED_ret_5d", "ASX_Australia_ret_5d", "PAYX_Paychex_ret_20d", "heston_var_ev_h3", "GE_ret_1d", "IYR_US_REIT2_zscore_60d", "EQIX_Equinix_ret_5d"], "is_new": true}, {"model_id": "new_h5_NORMAL_LightGBM_N20_t6", "algo": "LightGBM", "regime": "NORMAL", "horizon": 5, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "VOD_Vodafone_zscore_60d", "US5Y_Rate_ret_5d", "CCI_CrownCastle_vol_20d", "BTI_BritishAmerican_ret_5d", "XLF_Fin_vol_20d", "XOM_ret_20d", "PG_ret_20d", "Core_CPI_zscore_60d", "ITT_ITTInc_ret_5d", "SLB_Schlumberger_ret_1d", "AORD_AUS_zscore_60d", "SLB_Schlumberger_ret_5d", "MRK_Merck_zscore_60d", "DAX_Germany_vol_20d", "CI_Cigna_vol_20d", "BTI_BritishAmerican_ret_20d", "AMT_AmericanTower_ret_1d", "vix_acceleration_1d"], "is_new": true}, {"model_id": "new_h5_NORMAL_LightGBM_N20_t7", "algo": "LightGBM", "regime": "NORMAL", "horizon": 5, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EMR_Emerson_ret_20d", "JNJ_ret_1d", "BLK_BlackRock_zscore_60d", "Brent_Oil_FRED_ret_5d", "Core_CPI_zscore_60d", "ENB_EnbridgeInc_ret_1d", "SJM_JM_Smucker_ret_5d", "PCAR_PaccarInc_ret_5d", "IYM_BasicMaterials_ret_20d", "T10Y2Y_Spread_ret_5d", "EQR_Equity_ret_1d", "MS_MorganStanley_zscore_60d", "PLD_Prologis_ret_5d", "AVB_AvalonBay_zscore_60d", "EWA_Australia_zscore_60d", "US3M_Rate_vol_20d", "BDX_Becton_Dickinson_ret_20d", "VOD_Vodafone_zscore_60d"], "is_new": true}, {"model_id": "new_h5_NORMAL_LightGBM_N25_t0", "algo": "LightGBM", "regime": "NORMAL", "horizon": 5, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "XLV_Health_zscore_60d", "VOD_Vodafone_zscore_60d", "CPB_CampbellSoup_ret_20d", "HD_ret_20d", "XLB_Materials_zscore_60d", "EWQ_France_zscore_60d", "SCHW_Schwab_ret_5d", "CPB_CampbellSoup_zscore_60d", "WTI_Oil_FRED_zscore_60d", "CMCSA_ret_1d", "ASX_Australia_ret_5d", "GILD_Gilead_ret_20d", "MO_AltriaMG_ret_1d", "CTAS_Cintas_vol_20d", "HangSeng_HK_ret_5d", "DAX_Germany_vol_20d", "EXC_Exelon_ret_1d", "MSTR_Bitcoin3_ret_20d", "TM_Telephone_ret_1d", "SJM_JM_Smucker_ret_5d", "Core_PCE_zscore_60d", "heston_var_ev_h7", "INTC_ret_5d"], "is_new": true}, {"model_id": "new_h5_NORMAL_LightGBM_N25_t1", "algo": "LightGBM", "regime": "NORMAL", "horizon": 5, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "US1Y_Rate_ret_5d", "EWM_Malaysia_vol_20d", "HUM_Humana_ret_5d", "TM_Telephone_vol_20d", "EWQ_France_zscore_60d", "Industrial_Production_zscore_60d", "VVIX_ret_20d", "BTI_BritishAmerican_ret_20d", "EFFR_vol_20d", "PLD_Prologis_ret_5d", "IYR_US_REIT2_zscore_60d", "vix_acceleration_1d", "NOC_Northrop_ret_20d", "ES_Evergy_ret_1d", "HangSeng_HK_ret_5d", "QQQ_vol_20d", "BDX_Becton_Dickinson_ret_20d", "LMT_LockheedMartin_vol_20d", "EQR_Equity_ret_1d", "XOM_ret_20d", "EWA_Australia_zscore_60d", "HangSeng_HK_vol_20d", "XLK_Tech_zscore_60d"], "is_new": true}, {"model_id": "new_h5_NORMAL_LightGBM_N25_t2", "algo": "LightGBM", "regime": "NORMAL", "horizon": 5, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "T_ret_1d", "EWA_Australia_zscore_60d", "LOW_Lowes_ret_20d", "NVDA_vol_20d", "vix_acceleration_1d", "heston_var_ev_h7", "MSTR_Bitcoin3_ret_1d", "GE_ret_1d", "ASX_Australia_ret_5d", "gjr_condvar_h1", "heston_ev_h3", "MO_AltriaMG_ret_1d", "TED_Spread_zscore_60d", "IYM_BasicMaterials_ret_20d", "SLB_Schlumberger_ret_1d", "US1Y_Rate_ret_20d", "Brent_Oil_FRED_ret_20d", "CPB_CampbellSoup_ret_5d", "TXN_vol_20d", "MRK_Merck_zscore_60d", "LMT_LockheedMartin_ret_1d", "PAYX_Paychex_zscore_60d", "QQQ_vol_20d"], "is_new": true}, {"model_id": "new_h5_NORMAL_LightGBM_N25_t3", "algo": "LightGBM", "regime": "NORMAL", "horizon": 5, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "PAYX_Paychex_zscore_60d", "LOW_Lowes_ret_5d", "PPL_PPL_ret_1d", "Nikkei_Japan_vol_20d", "Nikkei_Japan_zscore_60d", "DE_Deere_ret_5d", "PG_ret_20d", "Brent_Oil_FRED_ret_20d", "BDX_Becton_Dickinson_ret_20d", "MS_MorganStanley_ret_1d", "CPB_CampbellSoup_vol_20d", "ASX_Australia_ret_5d", "HD_ret_20d", "EOG_EOGResources_vol_20d", "XOM_ret_20d", "EWY_Korea_ret_20d", "AMT_AmericanTower_ret_1d", "QQQ_vol_20d", "EWG_Germany_vol_20d", "LMT_LockheedMartin_ret_1d", "EWM_Malaysia_zscore_60d", "US30Y_Rate_ret_20d", "heston_ev_h3"], "is_new": true}, {"model_id": "new_h5_NORMAL_LightGBM_N25_t4", "algo": "LightGBM", "regime": "NORMAL", "horizon": 5, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "XOM_ret_1d", "PLD_Prologis_ret_5d", "TM_Telephone_ret_1d", "AMT_AmericanTower_ret_1d", "AMD_ret_1d", "vix_acceleration_1d", "US30Y_Rate_ret_20d", "MSTR_Bitcoin3_ret_5d", "Core_CPI_zscore_60d", "Nikkei_Japan_vol_20d", "XLB_Materials_zscore_60d", "3M_ret_5d", "BLK_BlackRock_zscore_60d", "T10Y2Y_Spread_ret_5d", "INTC_ret_5d", "EWS_Singapore_ret_5d", "EXC_Exelon_zscore_60d", "hmm_p_stress", "IWM_SmallCap_vol_20d", "INTC_ret_1d", "vix_mean_abs_ret_5d", "ORCL_zscore_60d", "HangSeng_HK_ret_1d"], "is_new": true}, {"model_id": "new_h5_NORMAL_LightGBM_N25_t5", "algo": "LightGBM", "regime": "NORMAL", "horizon": 5, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "HangSeng_HK_vol_20d", "EWQ_France_ret_20d", "VRP_ma5", "FedFunds_zscore_60d", "HD_ret_1d", "NFCI_ret_5d", "XLB_Materials_zscore_60d", "EOG_EOGResources_ret_5d", "M_Macys_vol_20d", "heston_var_ev_h3", "EOG_EOGResources_vol_20d", "NOC_Northrop_ret_20d", "EWL_Switzerland_vol_20d", "VVIX_ret_20d", "LLY_zscore_60d", "CPB_CampbellSoup_ret_20d", "US7Y_Rate_ret_20d", "T10Y2Y_Spread_ret_5d", "CPB_CampbellSoup_zscore_60d", "EWM_Malaysia_vol_20d", "TGT_Target_zscore_60d", "EWM_Malaysia_zscore_60d", "EWA_Australia_ret_1d"], "is_new": true}, {"model_id": "new_h5_NORMAL_LightGBM_N25_t6", "algo": "LightGBM", "regime": "NORMAL", "horizon": 5, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "Industrial_Production_zscore_60d", "EQR_Equity_ret_1d", "US3Y_Rate_ret_5d", "XLY_Disc_vol_20d", "EFFR_vol_20d", "PG_ret_20d", "hmm_p_stress", "PPL_PPL_ret_1d", "PCAR_PaccarInc_ret_5d", "CLX_Clorox_vol_20d", "EWA_Australia_ret_1d", "MO_AltriaMG_ret_1d", "TGT_Target_zscore_60d", "spx_momentum_3d", "XLB_Materials_zscore_60d", "NFCI_ret_5d", "Core_PCE_zscore_60d", "MS_MorganStanley_ret_5d", "AMT_AmericanTower_ret_1d", "BDX_Becton_Dickinson_ret_20d", "CCI_CrownCastle_vol_20d", "EWG_Germany_vol_20d", "CPB_CampbellSoup_vol_20d"], "is_new": true}, {"model_id": "new_h5_NORMAL_LightGBM_N25_t7", "algo": "LightGBM", "regime": "NORMAL", "horizon": 5, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "BTI_BritishAmerican_ret_5d", "TM_Telephone_vol_20d", "PPL_PPL_ret_1d", "spx_momentum_3d", "XLF_Fin_vol_20d", "PG_ret_20d", "TED_Spread_vol_20d", "heston_var_ev_h5", "US6M_Rate_ret_20d", "ENB_EnbridgeInc_ret_1d", "US3Y_Rate_ret_5d", "EWQ_France_zscore_60d", "BDX_Becton_Dickinson_ret_20d", "US5Y_Rate_ret_5d", "XOM_ret_1d", "PAYX_Paychex_zscore_60d", "spx_vol_5d", "DHR_vol_20d", "INTC_ret_5d", "US1Y_Rate_ret_20d", "HangSeng_HK_ret_1d", "AXP_Amex_vol_20d", "BA_ret_1d"], "is_new": true}, {"model_id": "new_h5_NORMAL_LightGBM_N30_t0", "algo": "LightGBM", "regime": "NORMAL", "horizon": 5, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "heston_ev_h3", "IYM_BasicMaterials_ret_20d", "AMZN_ret_5d", "US7Y_Rate_ret_20d", "GE_ret_1d", "NEE_NextEra_ret_20d", "MSTR_Bitcoin3_ret_5d", "MS_MorganStanley_zscore_60d", "3M_ret_5d", "IYR_US_REIT2_zscore_60d", "heston_var_ev_h3", "NOC_Northrop_ret_20d", "SPY_zscore_60d", "US3M_Rate_zscore_60d", "spx_vol_5d", "EQR_Equity_ret_1d", "LUV_SouthwestAir_ret_5d", "MO_AltriaMG_ret_1d", "EFFR_vol_20d", "SLB_Schlumberger_ret_1d", "T10Y2Y_Spread_ret_5d", "PCAR_PaccarInc_ret_5d", "EWC_Canada_zscore_60d", "PFE_ret_1d", "US30Y_Rate_ret_20d", "FedFunds_zscore_60d", "XLV_Health_zscore_60d", "PAYX_Paychex_ret_20d"], "is_new": true}, {"model_id": "new_h5_NORMAL_LightGBM_N30_t1", "algo": "LightGBM", "regime": "NORMAL", "horizon": 5, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "US6M_Rate_ret_20d", "AMZN_ret_5d", "EOG_EOGResources_vol_20d", "GD_GeneralDynamics_zscore_60d", "Core_PCE_zscore_60d", "NFCI_ret_5d", "CPB_CampbellSoup_vol_20d", "SCHW_Schwab_ret_5d", "PAYX_Paychex_vol_20d", "DE_Deere_vol_20d", "EWA_Australia_ret_1d", "SJM_JM_Smucker_ret_1d", "US3M_Rate_zscore_60d", "ORCL_zscore_60d", "EWM_Malaysia_vol_20d", "ASX_Australia_ret_5d", "XOM_ret_1d", "AMD_ret_5d", "IYM_BasicMaterials_ret_20d", "EWM_Malaysia_ret_1d", "EWA_Australia_zscore_60d", "BDX_Becton_Dickinson_ret_20d", "Core_CPI_zscore_60d", "EMR_Emerson_ret_20d", "WTI_Oil_FRED_zscore_60d", "DAX_Germany_vol_20d", "TED_Spread_vol_20d", "Nikkei_Japan_vol_20d"], "is_new": true}, {"model_id": "new_h5_NORMAL_LightGBM_N30_t2", "algo": "LightGBM", "regime": "NORMAL", "horizon": 5, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "TED_Spread_zscore_60d", "MRK_Merck_zscore_60d", "INTC_ret_5d", "AXP_Amex_ret_20d", "EOG_EOGResources_ret_5d", "EWG_Germany_vol_20d", "CI_Cigna_vol_20d", "EWY_Korea_ret_20d", "PAYX_Paychex_zscore_60d", "HUM_Humana_ret_5d", "GE_ret_1d", "SBUX_zscore_60d", "Core_PCE_zscore_60d", "LLY_zscore_60d", "MSTR_Bitcoin3_ret_1d", "LOW_Lowes_ret_20d", "BA_ret_1d", "EWL_Switzerland_vol_20d", "QQQ_vol_20d", "CPB_CampbellSoup_ret_5d", "ITT_ITTInc_ret_5d", "AORD_AUS_zscore_60d", "AMD_ret_1d", "SBUX_ret_5d", "EXC_Exelon_ret_1d", "US1Y_Rate_ret_5d", "SO_SouthernCo_ret_5d", "EWL_Switzerland_zscore_60d"], "is_new": true}, {"model_id": "new_h5_NORMAL_LightGBM_N30_t3", "algo": "LightGBM", "regime": "NORMAL", "horizon": 5, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "BDX_Becton_Dickinson_ret_20d", "CCI_CrownCastle_vol_20d", "DHR_vol_20d", "ASX_Australia_ret_5d", "spx_vol_5d", "CPB_CampbellSoup_zscore_60d", "INTC_ret_5d", "spx_abs_ret_max_5d", "Michigan_Sentiment_ret_20d", "EOG_EOGResources_vol_20d", "AXP_Amex_vol_20d", "SBUX_ret_5d", "NWL_Newell_ret_20d", "PPL_PPL_ret_1d", "US3Y_Rate_ret_5d", "LUV_SouthwestAir_ret_5d", "SCHW_Schwab_ret_5d", "INTC_ret_1d", "PG_ret_20d", "CPB_CampbellSoup_vol_20d", "SLB_Schlumberger_ret_1d", "Nikkei_Japan_zscore_60d", "PAYX_Paychex_vol_20d", "heston_var_ev_h5", "MO_AltriaMG_ret_1d", "AMGN_Amgen_ret_1d", "AMZN_ret_5d", "ORCL_vol_20d"], "is_new": true}, {"model_id": "new_h5_NORMAL_LightGBM_N30_t4", "algo": "LightGBM", "regime": "NORMAL", "horizon": 5, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "JNJ_ret_1d", "SO_SouthernCo_ret_5d", "PAYX_Paychex_vol_20d", "heston_var_ev_h5", "US7Y_Rate_ret_20d", "EXC_Exelon_zscore_60d", "VOD_Vodafone_zscore_60d", "DIS_vol_20d", "EWH_HongKong_ret_5d", "Core_PCE_zscore_60d", "gjr_condvar_h1", "LLY_zscore_60d", "CPB_CampbellSoup_zscore_60d", "CTAS_Cintas_vol_20d", "AMD_ret_1d", "US1Y_Rate_ret_5d", "CCI_CrownCastle_vol_20d", "SCHW_Schwab_ret_5d", "vix_acceleration_1d", "EWC_Canada_zscore_60d", "XLF_Fin_vol_20d", "EWM_Malaysia_vol_20d", "IYM_BasicMaterials_ret_20d", "PFE_ret_1d", "Brent_Oil_FRED_ret_20d", "BLK_BlackRock_zscore_60d", "VVIX_ret_20d", "NEE_NextEra_ret_20d"], "is_new": true}, {"model_id": "new_h5_NORMAL_LightGBM_N30_t5", "algo": "LightGBM", "regime": "NORMAL", "horizon": 5, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "LOW_Lowes_ret_5d", "SLB_Schlumberger_ret_1d", "vix_acceleration_1d", "CI_Cigna_vol_20d", "TED_Spread_vol_20d", "SLB_Schlumberger_ret_5d", "EWQ_France_zscore_60d", "PLD_Prologis_ret_5d", "MSTR_Bitcoin3_ret_1d", "AMT_AmericanTower_ret_1d", "NVDA_vol_20d", "BTI_BritishAmerican_ret_5d", "PAYX_Paychex_zscore_60d", "Brent_Oil_FRED_ret_5d", "DE_Deere_ret_5d", "LMT_LockheedMartin_vol_20d", "JNJ_ret_1d", "BDX_Becton_Dickinson_ret_20d", "XOM_ret_1d", "LLY_zscore_60d", "TM_Telephone_vol_20d", "FedFunds_zscore_60d", "US3Y_Rate_ret_5d", "EWC_Canada_zscore_60d", "Core_CPI_zscore_60d", "Retail_Sales_zscore_60d", "TM_Telephone_ret_1d", "EFFR_vol_20d"], "is_new": true}, {"model_id": "new_h5_NORMAL_LightGBM_N30_t6", "algo": "LightGBM", "regime": "NORMAL", "horizon": 5, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "GE_ret_1d", "spx_vol_5d", "vix_mean_abs_ret_5d", "HangSeng_HK_vol_20d", "EWH_HongKong_ret_5d", "GILD_Gilead_ret_20d", "BLK_BlackRock_zscore_60d", "EQR_Equity_ret_1d", "SPY_zscore_60d", "T10Y2Y_Spread_ret_5d", "IYR_US_REIT2_zscore_60d", "INTC_ret_1d", "VRP_ma5", "VVIX_ret_20d", "CI_Cigna_vol_20d", "BDX_Becton_Dickinson_ret_20d", "SLB_Schlumberger_ret_5d", "PCAR_PaccarInc_ret_5d", "PAYX_Paychex_zscore_60d", "CPB_CampbellSoup_zscore_60d", "SBUX_zscore_60d", "XOM_ret_20d", "ORCL_vol_20d", "MSTR_Bitcoin3_ret_1d", "EWY_Korea_ret_20d", "VOD_Vodafone_zscore_60d", "XOM_ret_1d", "3M_ret_5d"], "is_new": true}, {"model_id": "new_h5_NORMAL_LightGBM_N30_t7", "algo": "LightGBM", "regime": "NORMAL", "horizon": 5, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EMR_Emerson_ret_20d", "BLK_BlackRock_zscore_60d", "AMD_ret_1d", "PLD_Prologis_ret_5d", "EWG_Germany_vol_20d", "US1Y_Rate_ret_20d", "MS_MorganStanley_ret_5d", "IYM_BasicMaterials_ret_20d", "HD_zscore_60d", "EWM_Malaysia_zscore_60d", "PAYX_Paychex_ret_20d", "3M_ret_5d", "CPB_CampbellSoup_zscore_60d", "DAX_Germany_vol_20d", "DE_Deere_ret_5d", "heston_var_ev_h7", "VVIX_ret_20d", "US7Y_Rate_ret_20d", "EQIX_Equinix_ret_5d", "TED_Spread_vol_20d", "PFE_ret_1d", "DE_Deere_vol_20d", "VOD_Vodafone_zscore_60d", "FedFunds_zscore_60d", "US3Y_Rate_ret_5d", "gjr_condvar_h1", "NEE_NextEra_ret_20d", "CMCSA_ret_1d"], "is_new": true}, {"model_id": "new_h5_NORMAL_GradientBoosting_N5_t0", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 5, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "QQQ_vol_20d", "heston_var_ev_h5", "MSTR_Bitcoin3_ret_20d"], "is_new": true}, {"model_id": "new_h5_NORMAL_GradientBoosting_N5_t1", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 5, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "AMT_AmericanTower_ret_1d", "XOM_ret_1d", "TGT_Target_zscore_60d"], "is_new": true}, {"model_id": "new_h5_NORMAL_GradientBoosting_N5_t2", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 5, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWC_Canada_zscore_60d", "CCI_CrownCastle_vol_20d", "DAX_Germany_vol_20d"], "is_new": true}, {"model_id": "new_h5_NORMAL_GradientBoosting_N5_t3", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 5, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "MS_MorganStanley_ret_1d", "IYR_US_REIT2_zscore_60d", "DE_Deere_vol_20d"], "is_new": true}, {"model_id": "new_h5_NORMAL_GradientBoosting_N5_t4", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 5, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "Michigan_Sentiment_ret_20d", "TED_Spread_zscore_60d", "EXC_Exelon_zscore_60d"], "is_new": true}, {"model_id": "new_h5_NORMAL_GradientBoosting_N5_t5", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 5, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "NOC_Northrop_ret_20d", "LUV_SouthwestAir_ret_5d", "TM_Telephone_vol_20d"], "is_new": true}, {"model_id": "new_h5_NORMAL_GradientBoosting_N5_t6", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 5, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "heston_ev_h3", "US6M_Rate_ret_20d", "CI_Cigna_vol_20d"], "is_new": true}, {"model_id": "new_h5_NORMAL_GradientBoosting_N5_t7", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 5, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "GD_GeneralDynamics_zscore_60d", "US7Y_Rate_ret_20d", "PAYX_Paychex_zscore_60d"], "is_new": true}, {"model_id": "new_h5_NORMAL_GradientBoosting_N8_t0", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 5, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "NOC_Northrop_ret_20d", "SCHW_Schwab_ret_5d", "Brent_Oil_FRED_ret_20d", "EFFR_vol_20d", "CCI_CrownCastle_vol_20d", "EMR_Emerson_ret_20d"], "is_new": true}, {"model_id": "new_h5_NORMAL_GradientBoosting_N8_t1", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 5, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "Brent_Oil_FRED_ret_5d", "ITT_ITTInc_ret_5d", "LMT_LockheedMartin_vol_20d", "QQQ_vol_20d", "vix_mean_abs_ret_5d", "DE_Deere_vol_20d"], "is_new": true}, {"model_id": "new_h5_NORMAL_GradientBoosting_N8_t2", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 5, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWY_Korea_ret_20d", "US7Y_Rate_ret_20d", "MSTR_Bitcoin3_ret_1d", "TED_Spread_vol_20d", "EWQ_France_zscore_60d", "VRP_ma5"], "is_new": true}, {"model_id": "new_h5_NORMAL_GradientBoosting_N8_t3", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 5, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "US5Y_Rate_ret_5d", "PAYX_Paychex_zscore_60d", "Nikkei_Japan_zscore_60d", "SBUX_vol_20d", "EQIX_Equinix_ret_5d", "AMGN_Amgen_ret_1d"], "is_new": true}, {"model_id": "new_h5_NORMAL_GradientBoosting_N8_t4", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 5, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "LUV_SouthwestAir_ret_5d", "AORD_AUS_zscore_60d", "heston_var_ev_h5", "EWM_Malaysia_vol_20d", "EWA_Australia_ret_1d", "NEE_NextEra_ret_20d"], "is_new": true}, {"model_id": "new_h5_NORMAL_GradientBoosting_N8_t5", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 5, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "SCHW_Schwab_ret_5d", "US30Y_Rate_ret_20d", "EWG_Germany_vol_20d", "Nikkei_Japan_vol_20d", "QQQ_vol_20d", "SBUX_ret_5d"], "is_new": true}, {"model_id": "new_h5_NORMAL_GradientBoosting_N8_t6", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 5, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EOG_EOGResources_ret_5d", "HD_zscore_60d", "IYM_BasicMaterials_ret_20d", "PAYX_Paychex_zscore_60d", "Core_CPI_zscore_60d", "US3Y_Rate_ret_5d"], "is_new": true}, {"model_id": "new_h5_NORMAL_GradientBoosting_N8_t7", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 5, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "TM_Telephone_vol_20d", "EWY_Korea_zscore_60d", "QQQ_vol_20d", "EFFR_vol_20d", "PCAR_PaccarInc_ret_5d", "M_Macys_vol_20d"], "is_new": true}, {"model_id": "new_h5_NORMAL_GradientBoosting_N10_t0", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 5, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "TXN_vol_20d", "CMCSA_ret_1d", "Retail_Sales_zscore_60d", "SJM_JM_Smucker_ret_1d", "CTAS_Cintas_vol_20d", "SBUX_vol_20d", "MSTR_Bitcoin3_ret_5d", "US7Y_Rate_ret_20d"], "is_new": true}, {"model_id": "new_h5_NORMAL_GradientBoosting_N10_t1", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 5, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "ES_Evergy_ret_1d", "heston_var_ev_h7", "US30Y_Rate_ret_20d", "EWA_Australia_ret_1d", "US7Y_Rate_ret_20d", "MSTR_Bitcoin3_ret_1d", "EWJ_Japan_vol_20d", "EOG_EOGResources_vol_20d"], "is_new": true}, {"model_id": "new_h5_NORMAL_GradientBoosting_N10_t2", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 5, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "IBEX_Spain_ret_20d", "ENB_EnbridgeInc_ret_1d", "US7Y_Rate_ret_20d", "spx_momentum_3d", "MRK_Merck_zscore_60d", "ITT_ITTInc_ret_5d", "GE_ret_1d", "PPL_PPL_ret_1d"], "is_new": true}, {"model_id": "new_h5_NORMAL_GradientBoosting_N10_t3", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 5, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "ORCL_zscore_60d", "EWM_Malaysia_zscore_60d", "CLX_Clorox_vol_20d", "MSTR_Bitcoin3_ret_5d", "US1Y_Rate_ret_5d", "AVB_AvalonBay_zscore_60d", "TM_Telephone_ret_1d", "AXP_Amex_ret_20d"], "is_new": true}, {"model_id": "new_h5_NORMAL_GradientBoosting_N10_t4", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 5, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWS_Singapore_ret_5d", "XLY_Disc_vol_20d", "heston_ev_h3", "WTI_Oil_FRED_zscore_60d", "QQQ_vol_20d", "heston_var_ev_h3", "ORCL_zscore_60d", "MS_MorganStanley_ret_5d"], "is_new": true}, {"model_id": "new_h5_NORMAL_GradientBoosting_N10_t5", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 5, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "DAX_Germany_zscore_60d", "XOM_ret_1d", "US3M_Rate_zscore_60d", "TM_Telephone_ret_1d", "EWL_Switzerland_zscore_60d", "US5Y_Rate_ret_5d", "CPB_CampbellSoup_ret_5d", "US1Y_Rate_ret_20d"], "is_new": true}, {"model_id": "new_h5_NORMAL_GradientBoosting_N10_t6", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 5, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "SJM_JM_Smucker_ret_5d", "TED_Spread_zscore_60d", "HangSeng_HK_ret_1d", "Core_CPI_zscore_60d", "US7Y_Rate_ret_20d", "heston_var_ev_h7", "IBEX_Spain_ret_20d", "SO_SouthernCo_ret_5d"], "is_new": true}, {"model_id": "new_h5_NORMAL_GradientBoosting_N10_t7", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 5, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "Core_CPI_zscore_60d", "BTI_BritishAmerican_ret_5d", "TED_Spread_zscore_60d", "T10Y2Y_Spread_ret_5d", "3M_vol_20d", "EWM_Malaysia_zscore_60d", "GE_ret_1d", "M_Macys_vol_20d"], "is_new": true}, {"model_id": "new_h5_NORMAL_GradientBoosting_N12_t0", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 5, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "CPB_CampbellSoup_zscore_60d", "NWL_Newell_ret_20d", "XLB_Materials_zscore_60d", "EWJ_Japan_vol_20d", "TM_Telephone_vol_20d", "FedFunds_zscore_60d", "SBUX_zscore_60d", "EWQ_France_zscore_60d", "SLB_Schlumberger_ret_5d", "Brent_Oil_FRED_ret_20d"], "is_new": true}, {"model_id": "new_h5_NORMAL_GradientBoosting_N12_t1", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 5, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "3M_vol_20d", "ORCL_zscore_60d", "EOG_EOGResources_ret_5d", "Brent_Oil_FRED_ret_20d", "US30Y_Rate_ret_20d", "MRK_Merck_zscore_60d", "ASX_Australia_vol_20d", "DAX_Germany_zscore_60d", "DHR_ret_1d", "MS_MorganStanley_ret_5d"], "is_new": true}, {"model_id": "new_h5_NORMAL_GradientBoosting_N12_t2", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 5, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "AXP_Amex_ret_20d", "JNJ_ret_1d", "HD_ret_20d", "MS_MorganStanley_ret_1d", "HD_ret_5d", "XLK_Tech_zscore_60d", "SLB_Schlumberger_ret_1d", "IWM_SmallCap_vol_20d", "XLV_Health_zscore_60d", "AMT_AmericanTower_ret_1d"], "is_new": true}, {"model_id": "new_h5_NORMAL_GradientBoosting_N12_t3", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 5, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "IBEX_Spain_ret_20d", "BLK_BlackRock_zscore_60d", "AMD_ret_1d", "Brent_Oil_FRED_ret_20d", "PCAR_PaccarInc_ret_5d", "LUV_SouthwestAir_ret_5d", "LOW_Lowes_ret_5d", "AMZN_ret_5d", "HD_zscore_60d", "VRP_ma5"], "is_new": true}, {"model_id": "new_h5_NORMAL_GradientBoosting_N12_t4", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 5, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "T_ret_1d", "Core_CPI_zscore_60d", "HD_ret_5d", "CMCSA_ret_1d", "DOW_Price_zscore_60d", "PPL_PPL_ret_1d", "SBUX_vol_20d", "AMZN_ret_5d", "Brent_Oil_FRED_ret_5d", "LOW_Lowes_ret_5d"], "is_new": true}, {"model_id": "new_h5_NORMAL_GradientBoosting_N12_t5", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 5, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "spx_vol_5d", "TM_Telephone_ret_1d", "EWQ_France_zscore_60d", "VOD_Vodafone_zscore_60d", "EOG_EOGResources_ret_5d", "VVIX_ret_20d", "DAX_Germany_vol_20d", "EFFR_ret_1d", "SJM_JM_Smucker_ret_5d", "CCI_CrownCastle_vol_20d"], "is_new": true}, {"model_id": "new_h5_NORMAL_GradientBoosting_N12_t6", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 5, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "BDX_Becton_Dickinson_ret_20d", "SCHW_Schwab_ret_5d", "EWY_Korea_zscore_60d", "EWM_Malaysia_ret_1d", "DHR_vol_20d", "MSTR_Bitcoin3_ret_5d", "ENB_EnbridgeInc_ret_1d", "TM_Telephone_ret_1d", "gjr_condvar_h1", "EWJ_Japan_vol_20d"], "is_new": true}, {"model_id": "new_h5_NORMAL_GradientBoosting_N12_t7", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 5, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "AVB_AvalonBay_zscore_60d", "gjr_condvar_h1", "EWJ_Japan_vol_20d", "INTC_ret_5d", "MSTR_Bitcoin3_ret_1d", "QQQ_vol_20d", "DHR_ret_1d", "VRP_ma5", "HD_ret_1d", "GILD_Gilead_ret_20d"], "is_new": true}, {"model_id": "new_h5_NORMAL_GradientBoosting_N15_t0", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 5, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "DHR_ret_1d", "EFFR_vol_20d", "NWL_Newell_ret_20d", "HD_ret_5d", "Retail_Sales_zscore_60d", "DOW_Price_zscore_60d", "SJM_JM_Smucker_ret_5d", "EWY_Korea_ret_20d", "heston_var_ev_h5", "EWS_Singapore_ret_5d", "EWL_Switzerland_zscore_60d", "HangSeng_HK_vol_20d", "EWQ_France_ret_20d"], "is_new": true}, {"model_id": "new_h5_NORMAL_GradientBoosting_N15_t1", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 5, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EQR_Equity_ret_1d", "DE_Deere_vol_20d", "DOW_Price_zscore_60d", "spx_momentum_3d", "PAYX_Paychex_vol_20d", "IYM_BasicMaterials_ret_20d", "MSTR_Bitcoin3_ret_1d", "CPB_CampbellSoup_ret_5d", "SJM_JM_Smucker_ret_1d", "hmm_p_stress", "LMT_LockheedMartin_vol_20d", "TED_Spread_zscore_60d", "EWJ_Japan_vol_20d"], "is_new": true}, {"model_id": "new_h5_NORMAL_GradientBoosting_N15_t2", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 5, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWL_Switzerland_zscore_60d", "AMZN_ret_5d", "XOM_ret_1d", "SLB_Schlumberger_ret_1d", "CI_Cigna_vol_20d", "SO_SouthernCo_ret_5d", "US5Y_Rate_ret_5d", "DHR_vol_20d", "HD_ret_5d", "spx_momentum_3d", "HD_ret_1d", "CTAS_Cintas_vol_20d", "EOG_EOGResources_ret_5d"], "is_new": true}, {"model_id": "new_h5_NORMAL_GradientBoosting_N15_t3", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 5, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "XLB_Materials_zscore_60d", "SCHW_Schwab_ret_5d", "MO_AltriaMG_ret_1d", "EMR_Emerson_ret_20d", "DHR_ret_1d", "SO_SouthernCo_ret_5d", "SLB_Schlumberger_ret_5d", "EWM_Malaysia_vol_20d", "DOW_Price_zscore_60d", "AORD_AUS_zscore_60d", "XLY_Disc_vol_20d", "EWL_Switzerland_zscore_60d", "VOD_Vodafone_zscore_60d"], "is_new": true}, {"model_id": "new_h5_NORMAL_GradientBoosting_N15_t4", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 5, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "XLF_Fin_vol_20d", "EQIX_Equinix_ret_5d", "XLK_Tech_zscore_60d", "EWM_Malaysia_zscore_60d", "NEE_NextEra_ret_20d", "PAYX_Paychex_ret_20d", "ORCL_zscore_60d", "TGT_Target_zscore_60d", "MO_AltriaMG_ret_1d", "CI_Cigna_vol_20d", "GILD_Gilead_ret_20d", "LMT_LockheedMartin_ret_1d", "JNJ_ret_1d"], "is_new": true}, {"model_id": "new_h5_NORMAL_GradientBoosting_N15_t5", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 5, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "HangSeng_HK_vol_20d", "DHR_vol_20d", "WTI_Oil_FRED_zscore_60d", "heston_var_ev_h7", "PG_ret_20d", "EWL_Switzerland_zscore_60d", "SBUX_zscore_60d", "DIS_vol_20d", "Nikkei_Japan_vol_20d", "US30Y_Rate_ret_20d", "EWA_Australia_ret_1d", "EMR_Emerson_ret_20d", "EWM_Malaysia_zscore_60d"], "is_new": true}, {"model_id": "new_h5_NORMAL_GradientBoosting_N15_t6", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 5, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "SBUX_vol_20d", "LMT_LockheedMartin_vol_20d", "MRK_Merck_zscore_60d", "PAYX_Paychex_vol_20d", "heston_var_ev_h3", "gjr_condvar_h1", "EWL_Switzerland_zscore_60d", "SCHW_Schwab_ret_5d", "TM_Telephone_vol_20d", "JNJ_ret_1d", "CMCSA_ret_1d", "EWM_Malaysia_zscore_60d", "VVIX_ret_20d"], "is_new": true}, {"model_id": "new_h5_NORMAL_GradientBoosting_N15_t7", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 5, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "IYM_BasicMaterials_ret_20d", "EWL_Switzerland_vol_20d", "Core_CPI_zscore_60d", "DIS_vol_20d", "NVDA_vol_20d", "EWY_Korea_zscore_60d", "LLY_zscore_60d", "MSTR_Bitcoin3_ret_20d", "XLV_Health_zscore_60d", "MSTR_Bitcoin3_ret_1d", "PAYX_Paychex_ret_20d", "LMT_LockheedMartin_vol_20d", "ASX_Australia_ret_5d"], "is_new": true}, {"model_id": "new_h5_NORMAL_GradientBoosting_N20_t0", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 5, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "CMCSA_ret_1d", "LOW_Lowes_ret_5d", "XOM_ret_1d", "heston_var_ev_h7", "CLX_Clorox_vol_20d", "hmm_p_stress", "TM_Telephone_vol_20d", "EWM_Malaysia_ret_1d", "EWM_Malaysia_vol_20d", "EWM_Malaysia_zscore_60d", "LUV_SouthwestAir_ret_5d", "MS_MorganStanley_zscore_60d", "ENB_EnbridgeInc_ret_1d", "gjr_condvar_h1", "spx_abs_ret_max_5d", "MRK_Merck_zscore_60d", "Industrial_Production_zscore_60d", "AMGN_Amgen_ret_1d"], "is_new": true}, {"model_id": "new_h5_NORMAL_GradientBoosting_N20_t1", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 5, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "CTAS_Cintas_vol_20d", "BDX_Becton_Dickinson_ret_20d", "EWS_Singapore_ret_5d", "EXC_Exelon_zscore_60d", "CI_Cigna_vol_20d", "Brent_Oil_FRED_ret_5d", "EWM_Malaysia_zscore_60d", "ORCL_zscore_60d", "EWA_Australia_zscore_60d", "spx_vol_5d", "DHR_ret_1d", "AMZN_ret_5d", "heston_ev_h3", "spx_abs_ret_max_5d", "DAX_Germany_zscore_60d", "DE_Deere_ret_5d", "ASX_Australia_ret_5d", "PCAR_PaccarInc_ret_5d"], "is_new": true}, {"model_id": "new_h5_NORMAL_GradientBoosting_N20_t2", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 5, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "AVB_AvalonBay_zscore_60d", "MS_MorganStanley_zscore_60d", "TM_Telephone_ret_1d", "MS_MorganStanley_ret_5d", "gjr_condvar_h1", "US1Y_Rate_ret_5d", "heston_var_ev_h5", "NFCI_ret_5d", "NVDA_vol_20d", "EWM_Malaysia_vol_20d", "3M_ret_5d", "SBUX_ret_5d", "AMGN_Amgen_ret_1d", "XOM_ret_1d", "HD_ret_20d", "TXN_vol_20d", "TM_Telephone_vol_20d", "NEE_NextEra_ret_20d"], "is_new": true}, {"model_id": "new_h5_NORMAL_GradientBoosting_N20_t3", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 5, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "US30Y_Rate_ret_20d", "CPB_CampbellSoup_ret_5d", "US5Y_Rate_ret_5d", "EOG_EOGResources_ret_5d", "XOM_ret_1d", "US3M_Rate_zscore_60d", "IYM_BasicMaterials_ret_20d", "US3Y_Rate_ret_5d", "CCI_CrownCastle_vol_20d", "CLX_Clorox_vol_20d", "AMGN_Amgen_ret_1d", "DAX_Germany_zscore_60d", "XLY_Disc_vol_20d", "TED_Spread_vol_20d", "HangSeng_HK_vol_20d", "US6M_Rate_ret_20d", "hmm_p_stress", "EWH_HongKong_ret_5d"], "is_new": true}, {"model_id": "new_h5_NORMAL_GradientBoosting_N20_t4", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 5, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "US3M_Rate_zscore_60d", "EWS_Singapore_ret_5d", "DIS_vol_20d", "PAYX_Paychex_ret_20d", "MO_AltriaMG_ret_1d", "EWA_Australia_ret_1d", "DAX_Germany_zscore_60d", "XLY_Disc_vol_20d", "MS_MorganStanley_ret_1d", "XLF_Fin_vol_20d", "HD_zscore_60d", "SJM_JM_Smucker_ret_5d", "MS_MorganStanley_zscore_60d", "spx_vol_5d", "BDX_Becton_Dickinson_ret_20d", "CCI_CrownCastle_vol_20d", "AMD_ret_5d", "ITT_ITTInc_ret_5d"], "is_new": true}, {"model_id": "new_h5_NORMAL_GradientBoosting_N20_t5", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 5, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "WTI_Oil_FRED_zscore_60d", "MS_MorganStanley_ret_5d", "XLF_Fin_vol_20d", "Brent_Oil_FRED_ret_20d", "CPB_CampbellSoup_ret_5d", "spx_vol_5d", "US1Y_Rate_ret_5d", "MSTR_Bitcoin3_ret_1d", "MSTR_Bitcoin3_ret_20d", "IBEX_Spain_ret_20d", "CTAS_Cintas_vol_20d", "Industrial_Production_zscore_60d", "CMCSA_ret_1d", "MSTR_Bitcoin3_ret_5d", "HUM_Humana_ret_5d", "vix_mean_abs_ret_5d", "EWY_Korea_zscore_60d", "NFCI_ret_5d"], "is_new": true}, {"model_id": "new_h5_NORMAL_GradientBoosting_N20_t6", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 5, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "MRK_Merck_zscore_60d", "VVIX_ret_20d", "PPL_PPL_ret_1d", "US30Y_Rate_ret_20d", "DOW_Price_zscore_60d", "Nikkei_Japan_zscore_60d", "US7Y_Rate_ret_20d", "AMGN_Amgen_ret_1d", "BA_ret_1d", "TED_Spread_vol_20d", "AMD_ret_5d", "3M_ret_5d", "HD_ret_20d", "PCAR_PaccarInc_ret_5d", "EWC_Canada_zscore_60d", "ES_Evergy_ret_1d", "EMR_Emerson_ret_20d", "NOC_Northrop_ret_20d"], "is_new": true}, {"model_id": "new_h5_NORMAL_GradientBoosting_N20_t7", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 5, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "US3M_Rate_vol_20d", "CCI_CrownCastle_vol_20d", "EWG_Germany_ret_20d", "GD_GeneralDynamics_zscore_60d", "XLY_Disc_vol_20d", "PFE_ret_1d", "EWS_Singapore_ret_5d", "Core_PCE_zscore_60d", "AMT_AmericanTower_ret_1d", "LUV_SouthwestAir_ret_5d", "EWJ_Japan_vol_20d", "US30Y_Rate_ret_20d", "SPY_zscore_60d", "TED_Spread_zscore_60d", "heston_var_ev_h7", "spx_momentum_3d", "WTI_Oil_FRED_zscore_60d", "EWL_Switzerland_zscore_60d"], "is_new": true}, {"model_id": "new_h5_NORMAL_GradientBoosting_N25_t0", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 5, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "GE_ret_1d", "EMR_Emerson_ret_20d", "SJM_JM_Smucker_ret_5d", "DIS_vol_20d", "EWQ_France_ret_20d", "NVDA_vol_20d", "DAX_Germany_vol_20d", "Nikkei_Japan_zscore_60d", "ASX_Australia_ret_5d", "CI_Cigna_vol_20d", "AMT_AmericanTower_ret_1d", "XOM_ret_1d", "SBUX_zscore_60d", "spx_momentum_3d", "US30Y_Rate_ret_20d", "EWG_Germany_vol_20d", "TGT_Target_zscore_60d", "EOG_EOGResources_ret_5d", "BTI_BritishAmerican_ret_5d", "DAX_Germany_zscore_60d", "CPB_CampbellSoup_ret_20d", "AVB_AvalonBay_zscore_60d", "VRP_ma5"], "is_new": true}, {"model_id": "new_h5_NORMAL_GradientBoosting_N25_t1", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 5, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "AXP_Amex_vol_20d", "WTI_Oil_FRED_zscore_60d", "SBUX_zscore_60d", "SO_SouthernCo_ret_5d", "HangSeng_HK_ret_5d", "US3M_Rate_vol_20d", "SBUX_ret_5d", "AMZN_ret_5d", "HD_ret_20d", "EFFR_ret_1d", "ASX_Australia_ret_5d", "US6M_Rate_ret_20d", "ORCL_zscore_60d", "PFE_ret_1d", "Michigan_Sentiment_ret_20d", "Nikkei_Japan_vol_20d", "EWQ_France_zscore_60d", "CPB_CampbellSoup_vol_20d", "gjr_condvar_h1", "vix_mean_abs_ret_5d", "IBEX_Spain_ret_20d", "IYR_US_REIT2_zscore_60d", "EWG_Germany_ret_20d"], "is_new": true}, {"model_id": "new_h5_NORMAL_GradientBoosting_N25_t2", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 5, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWJ_Japan_vol_20d", "MSTR_Bitcoin3_ret_5d", "IWM_SmallCap_vol_20d", "EWQ_France_zscore_60d", "ENB_EnbridgeInc_ret_1d", "QQQ_vol_20d", "CCI_CrownCastle_vol_20d", "EXC_Exelon_zscore_60d", "SBUX_ret_5d", "EQIX_Equinix_ret_5d", "LLY_zscore_60d", "AMD_ret_5d", "heston_var_ev_h3", "MSTR_Bitcoin3_ret_1d", "EFFR_ret_1d", "EWC_Canada_zscore_60d", "EWM_Malaysia_vol_20d", "DAX_Germany_zscore_60d", "PFE_ret_1d", "MRK_Merck_zscore_60d", "AVB_AvalonBay_zscore_60d", "US6M_Rate_ret_20d", "GE_ret_1d"], "is_new": true}, {"model_id": "new_h5_NORMAL_GradientBoosting_N25_t3", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 5, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "SCHW_Schwab_ret_5d", "vix_acceleration_1d", "HD_ret_5d", "US6M_Rate_ret_20d", "EWJ_Japan_vol_20d", "NOC_Northrop_ret_20d", "AMD_ret_1d", "Brent_Oil_FRED_ret_20d", "MS_MorganStanley_zscore_60d", "CCI_CrownCastle_vol_20d", "ORCL_vol_20d", "SO_SouthernCo_ret_5d", "QQQ_vol_20d", "MSTR_Bitcoin3_ret_5d", "EWM_Malaysia_ret_1d", "AORD_AUS_zscore_60d", "SBUX_vol_20d", "EWG_Germany_ret_20d", "PLD_Prologis_ret_5d", "EMR_Emerson_ret_20d", "DOW_Price_zscore_60d", "INTC_ret_1d", "SBUX_ret_5d"], "is_new": true}, {"model_id": "new_h5_NORMAL_GradientBoosting_N25_t4", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 5, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWH_HongKong_ret_5d", "WTI_Oil_FRED_zscore_60d", "Core_PCE_zscore_60d", "PAYX_Paychex_ret_20d", "ASX_Australia_ret_5d", "CPB_CampbellSoup_vol_20d", "EWJ_Japan_vol_20d", "SO_SouthernCo_ret_5d", "XLV_Health_zscore_60d", "US1Y_Rate_ret_5d", "TXN_vol_20d", "hmm_p_stress", "3M_vol_20d", "DHR_vol_20d", "US3M_Rate_zscore_60d", "ITT_ITTInc_ret_5d", "spx_momentum_3d", "heston_ev_h3", "T10Y2Y_Spread_ret_5d", "DAX_Germany_zscore_60d", "EOG_EOGResources_vol_20d", "vix_mean_abs_ret_5d", "EWM_Malaysia_zscore_60d"], "is_new": true}, {"model_id": "new_h5_NORMAL_GradientBoosting_N25_t5", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 5, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWJ_Japan_vol_20d", "GILD_Gilead_ret_20d", "DHR_vol_20d", "spx_momentum_3d", "DE_Deere_ret_5d", "EWM_Malaysia_vol_20d", "HD_zscore_60d", "spx_abs_ret_max_5d", "spx_vol_5d", "NOC_Northrop_ret_20d", "HangSeng_HK_ret_5d", "Brent_Oil_FRED_ret_5d", "AMGN_Amgen_ret_1d", "DAX_Germany_vol_20d", "EXC_Exelon_zscore_60d", "vix_acceleration_1d", "heston_var_ev_h3", "EWA_Australia_zscore_60d", "3M_vol_20d", "LOW_Lowes_ret_5d", "MSTR_Bitcoin3_ret_20d", "MS_MorganStanley_ret_1d", "Industrial_Production_zscore_60d"], "is_new": true}, {"model_id": "new_h5_NORMAL_GradientBoosting_N25_t6", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 5, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "Michigan_Sentiment_ret_20d", "CPB_CampbellSoup_zscore_60d", "gjr_condvar_h1", "heston_var_ev_h3", "DOW_Price_zscore_60d", "LOW_Lowes_ret_20d", "DAX_Germany_vol_20d", "SBUX_vol_20d", "EXC_Exelon_zscore_60d", "EWQ_France_ret_20d", "HangSeng_HK_ret_5d", "EOG_EOGResources_vol_20d", "heston_var_ev_h7", "AORD_AUS_zscore_60d", "TM_Telephone_vol_20d", "IBEX_Spain_ret_20d", "heston_ev_h3", "TM_Telephone_ret_1d", "SO_SouthernCo_ret_5d", "US3M_Rate_vol_20d", "EWG_Germany_ret_20d", "US3M_Rate_zscore_60d", "AMZN_ret_5d"], "is_new": true}, {"model_id": "new_h5_NORMAL_GradientBoosting_N25_t7", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 5, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EXC_Exelon_zscore_60d", "MS_MorganStanley_ret_1d", "Nikkei_Japan_zscore_60d", "SPY_zscore_60d", "AMD_ret_1d", "DOW_Price_zscore_60d", "MS_MorganStanley_ret_5d", "US1Y_Rate_ret_5d", "QQQ_vol_20d", "HD_zscore_60d", "US30Y_Rate_ret_20d", "LLY_zscore_60d", "CTAS_Cintas_vol_20d", "CPB_CampbellSoup_vol_20d", "PAYX_Paychex_vol_20d", "NWL_Newell_ret_20d", "Nikkei_Japan_vol_20d", "Core_CPI_zscore_60d", "DIS_vol_20d", "HD_ret_1d", "SBUX_zscore_60d", "MSTR_Bitcoin3_ret_1d", "US1Y_Rate_ret_20d"], "is_new": true}, {"model_id": "new_h5_NORMAL_GradientBoosting_N30_t0", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 5, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "HangSeng_HK_ret_5d", "Nikkei_Japan_vol_20d", "US3M_Rate_vol_20d", "EFFR_vol_20d", "Nikkei_Japan_zscore_60d", "MO_AltriaMG_ret_1d", "WTI_Oil_FRED_zscore_60d", "heston_var_ev_h3", "AVB_AvalonBay_zscore_60d", "EQR_Equity_ret_1d", "T_ret_1d", "spx_momentum_3d", "QQQ_vol_20d", "PCAR_PaccarInc_ret_5d", "HangSeng_HK_ret_1d", "NFCI_ret_5d", "HD_ret_1d", "TXN_vol_20d", "heston_ev_h3", "IYR_US_REIT2_zscore_60d", "HD_zscore_60d", "INTC_ret_1d", "PLD_Prologis_ret_5d", "MSTR_Bitcoin3_ret_20d", "EWS_Singapore_ret_5d", "HUM_Humana_ret_5d", "US3Y_Rate_ret_5d", "PAYX_Paychex_zscore_60d"], "is_new": true}, {"model_id": "new_h5_NORMAL_GradientBoosting_N30_t1", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 5, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "CPB_CampbellSoup_zscore_60d", "EWA_Australia_zscore_60d", "SBUX_zscore_60d", "MSTR_Bitcoin3_ret_5d", "SLB_Schlumberger_ret_5d", "EWY_Korea_ret_20d", "LOW_Lowes_ret_5d", "EFFR_vol_20d", "T10Y2Y_Spread_ret_5d", "Brent_Oil_FRED_ret_5d", "EWQ_France_ret_20d", "SBUX_vol_20d", "DE_Deere_ret_5d", "FedFunds_zscore_60d", "EOG_EOGResources_vol_20d", "AXP_Amex_ret_20d", "HD_ret_5d", "NVDA_vol_20d", "LUV_SouthwestAir_ret_5d", "EWM_Malaysia_vol_20d", "INTC_ret_1d", "heston_var_ev_h5", "BTI_BritishAmerican_ret_5d", "SCHW_Schwab_ret_5d", "BLK_BlackRock_zscore_60d", "Industrial_Production_zscore_60d", "QQQ_vol_20d", "DE_Deere_vol_20d"], "is_new": true}, {"model_id": "new_h5_NORMAL_GradientBoosting_N30_t2", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 5, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "AXP_Amex_ret_20d", "AXP_Amex_vol_20d", "DIS_vol_20d", "vix_acceleration_1d", "EWA_Australia_zscore_60d", "ASX_Australia_vol_20d", "GD_GeneralDynamics_zscore_60d", "EWA_Australia_ret_1d", "EWG_Germany_ret_20d", "spx_momentum_3d", "SBUX_vol_20d", "PG_ret_20d", "US3M_Rate_vol_20d", "AMGN_Amgen_ret_1d", "AORD_AUS_zscore_60d", "CMCSA_ret_1d", "DE_Deere_ret_5d", "CLX_Clorox_vol_20d", "LOW_Lowes_ret_5d", "spx_abs_ret_max_5d", "EQR_Equity_ret_1d", "EWY_Korea_ret_20d", "SJM_JM_Smucker_ret_5d", "TXN_vol_20d", "IBEX_Spain_ret_20d", "BLK_BlackRock_zscore_60d", "US3M_Rate_zscore_60d", "Industrial_Production_zscore_60d"], "is_new": true}, {"model_id": "new_h5_NORMAL_GradientBoosting_N30_t3", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 5, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EMR_Emerson_ret_20d", "SBUX_vol_20d", "DAX_Germany_vol_20d", "US1Y_Rate_ret_5d", "spx_vol_5d", "BA_ret_1d", "MSTR_Bitcoin3_ret_5d", "EWG_Germany_ret_20d", "Brent_Oil_FRED_ret_20d", "VVIX_ret_20d", "TGT_Target_zscore_60d", "LUV_SouthwestAir_ret_5d", "HD_ret_20d", "GILD_Gilead_ret_20d", "TXN_vol_20d", "hmm_p_stress", "Nikkei_Japan_zscore_60d", "MS_MorganStanley_ret_1d", "PAYX_Paychex_zscore_60d", "MSTR_Bitcoin3_ret_20d", "GE_ret_1d", "XLF_Fin_vol_20d", "DHR_ret_1d", "ASX_Australia_ret_5d", "NVDA_vol_20d", "ITT_ITTInc_ret_5d", "BTI_BritishAmerican_ret_20d", "M_Macys_vol_20d"], "is_new": true}, {"model_id": "new_h5_NORMAL_GradientBoosting_N30_t4", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 5, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "INTC_ret_5d", "EWY_Korea_zscore_60d", "QQQ_vol_20d", "EWC_Canada_zscore_60d", "SBUX_ret_5d", "ASX_Australia_vol_20d", "GE_ret_1d", "ENB_EnbridgeInc_ret_1d", "BTI_BritishAmerican_ret_20d", "HangSeng_HK_ret_5d", "Core_PCE_zscore_60d", "LUV_SouthwestAir_ret_5d", "DHR_vol_20d", "EWQ_France_ret_20d", "US1Y_Rate_ret_5d", "PAYX_Paychex_zscore_60d", "ORCL_zscore_60d", "TED_Spread_zscore_60d", "SCHW_Schwab_ret_5d", "BLK_BlackRock_zscore_60d", "AMZN_ret_5d", "3M_vol_20d", "US3M_Rate_zscore_60d", "XLB_Materials_zscore_60d", "JNJ_ret_1d", "heston_var_ev_h7", "LLY_zscore_60d", "BDX_Becton_Dickinson_ret_20d"], "is_new": true}, {"model_id": "new_h5_NORMAL_GradientBoosting_N30_t5", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 5, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "M_Macys_vol_20d", "EWQ_France_zscore_60d", "EWG_Germany_vol_20d", "SO_SouthernCo_ret_5d", "spx_abs_ret_max_5d", "Core_CPI_zscore_60d", "ES_Evergy_ret_1d", "EWA_Australia_zscore_60d", "ITT_ITTInc_ret_5d", "HangSeng_HK_ret_1d", "heston_var_ev_h5", "EWJ_Japan_vol_20d", "Nikkei_Japan_zscore_60d", "AXP_Amex_ret_20d", "MS_MorganStanley_ret_5d", "EFFR_ret_1d", "3M_vol_20d", "spx_vol_5d", "SCHW_Schwab_ret_5d", "NFCI_ret_5d", "BDX_Becton_Dickinson_ret_20d", "DE_Deere_ret_5d", "ASX_Australia_vol_20d", "SJM_JM_Smucker_ret_1d", "ASX_Australia_ret_5d", "PAYX_Paychex_vol_20d", "EWY_Korea_ret_20d", "PPL_PPL_ret_1d"], "is_new": true}, {"model_id": "new_h5_NORMAL_GradientBoosting_N30_t6", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 5, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWQ_France_zscore_60d", "CMCSA_ret_1d", "ORCL_vol_20d", "XLF_Fin_vol_20d", "heston_var_ev_h5", "PAYX_Paychex_ret_20d", "spx_vol_5d", "AMGN_Amgen_ret_1d", "PAYX_Paychex_vol_20d", "spx_momentum_3d", "AMD_ret_5d", "GD_GeneralDynamics_zscore_60d", "MSTR_Bitcoin3_ret_5d", "Nikkei_Japan_zscore_60d", "AMT_AmericanTower_ret_1d", "AXP_Amex_ret_20d", "Industrial_Production_zscore_60d", "IYR_US_REIT2_zscore_60d", "VRP_ma5", "EWQ_France_ret_20d", "BTI_BritishAmerican_ret_20d", "US7Y_Rate_ret_20d", "US6M_Rate_ret_20d", "HD_ret_1d", "US3Y_Rate_ret_5d", "EWA_Australia_ret_1d", "GILD_Gilead_ret_20d", "XOM_ret_20d"], "is_new": true}, {"model_id": "new_h5_NORMAL_GradientBoosting_N30_t7", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 5, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "LUV_SouthwestAir_ret_5d", "XLY_Disc_vol_20d", "AXP_Amex_ret_20d", "ES_Evergy_ret_1d", "US30Y_Rate_ret_20d", "ORCL_zscore_60d", "XLK_Tech_zscore_60d", "spx_abs_ret_max_5d", "DOW_Price_zscore_60d", "NOC_Northrop_ret_20d", "EOG_EOGResources_vol_20d", "DHR_ret_1d", "IWM_SmallCap_vol_20d", "Industrial_Production_zscore_60d", "IYR_US_REIT2_zscore_60d", "PAYX_Paychex_vol_20d", "HangSeng_HK_ret_5d", "CPB_CampbellSoup_zscore_60d", "EXC_Exelon_zscore_60d", "LLY_zscore_60d", "spx_vol_5d", "T10Y2Y_Spread_ret_5d", "ASX_Australia_vol_20d", "ORCL_vol_20d", "MSTR_Bitcoin3_ret_5d", "AMZN_ret_5d", "EWM_Malaysia_zscore_60d", "heston_ev_h3"], "is_new": true}, {"model_id": "new_h5_NORMAL_RandomForest_N5_t0", "algo": "RandomForest", "regime": "NORMAL", "horizon": 5, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "MSTR_Bitcoin3_ret_20d", "CCI_CrownCastle_vol_20d", "NOC_Northrop_ret_20d"], "is_new": true}, {"model_id": "new_h5_NORMAL_RandomForest_N5_t1", "algo": "RandomForest", "regime": "NORMAL", "horizon": 5, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWC_Canada_zscore_60d", "EWY_Korea_ret_20d", "MSTR_Bitcoin3_ret_1d"], "is_new": true}, {"model_id": "new_h5_NORMAL_RandomForest_N5_t2", "algo": "RandomForest", "regime": "NORMAL", "horizon": 5, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "TGT_Target_zscore_60d", "AORD_AUS_zscore_60d", "GD_GeneralDynamics_zscore_60d"], "is_new": true}, {"model_id": "new_h5_NORMAL_RandomForest_N5_t3", "algo": "RandomForest", "regime": "NORMAL", "horizon": 5, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "MSTR_Bitcoin3_ret_5d", "EXC_Exelon_ret_1d", "HD_ret_1d"], "is_new": true}, {"model_id": "new_h5_NORMAL_RandomForest_N5_t4", "algo": "RandomForest", "regime": "NORMAL", "horizon": 5, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "HangSeng_HK_ret_5d", "US7Y_Rate_ret_20d", "GE_ret_1d"], "is_new": true}, {"model_id": "new_h5_NORMAL_RandomForest_N5_t5", "algo": "RandomForest", "regime": "NORMAL", "horizon": 5, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "heston_ev_h3", "SLB_Schlumberger_ret_1d", "NEE_NextEra_ret_20d"], "is_new": true}, {"model_id": "new_h5_NORMAL_RandomForest_N5_t6", "algo": "RandomForest", "regime": "NORMAL", "horizon": 5, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "LOW_Lowes_ret_5d", "NFCI_ret_5d", "PLD_Prologis_ret_5d"], "is_new": true}, {"model_id": "new_h5_NORMAL_RandomForest_N5_t7", "algo": "RandomForest", "regime": "NORMAL", "horizon": 5, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "US3Y_Rate_ret_5d", "CPB_CampbellSoup_ret_20d", "ORCL_zscore_60d"], "is_new": true}, {"model_id": "new_h5_NORMAL_RandomForest_N8_t0", "algo": "RandomForest", "regime": "NORMAL", "horizon": 5, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "vix_mean_abs_ret_5d", "TM_Telephone_vol_20d", "CPB_CampbellSoup_ret_5d", "DHR_ret_1d", "LLY_zscore_60d", "ORCL_zscore_60d"], "is_new": true}, {"model_id": "new_h5_NORMAL_RandomForest_N8_t1", "algo": "RandomForest", "regime": "NORMAL", "horizon": 5, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "XLV_Health_zscore_60d", "TED_Spread_zscore_60d", "ENB_EnbridgeInc_ret_1d", "DIS_vol_20d", "FedFunds_zscore_60d", "T_ret_1d"], "is_new": true}, {"model_id": "new_h5_NORMAL_RandomForest_N8_t2", "algo": "RandomForest", "regime": "NORMAL", "horizon": 5, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "QQQ_vol_20d", "BA_ret_1d", "NEE_NextEra_ret_20d", "SBUX_vol_20d", "LMT_LockheedMartin_ret_1d", "XLF_Fin_vol_20d"], "is_new": true}, {"model_id": "new_h5_NORMAL_RandomForest_N8_t3", "algo": "RandomForest", "regime": "NORMAL", "horizon": 5, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "spx_momentum_3d", "AMGN_Amgen_ret_1d", "EQR_Equity_ret_1d", "CTAS_Cintas_vol_20d", "US3M_Rate_zscore_60d", "PAYX_Paychex_zscore_60d"], "is_new": true}, {"model_id": "new_h5_NORMAL_RandomForest_N8_t4", "algo": "RandomForest", "regime": "NORMAL", "horizon": 5, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "heston_var_ev_h7", "PAYX_Paychex_ret_20d", "3M_vol_20d", "hmm_p_stress", "IBEX_Spain_ret_20d", "US1Y_Rate_ret_5d"], "is_new": true}, {"model_id": "new_h5_NORMAL_RandomForest_N8_t5", "algo": "RandomForest", "regime": "NORMAL", "horizon": 5, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "JNJ_ret_1d", "MS_MorganStanley_zscore_60d", "BTI_BritishAmerican_ret_5d", "DHR_vol_20d", "IBEX_Spain_ret_20d", "ORCL_vol_20d"], "is_new": true}, {"model_id": "new_h5_NORMAL_RandomForest_N8_t6", "algo": "RandomForest", "regime": "NORMAL", "horizon": 5, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "US6M_Rate_ret_20d", "LLY_zscore_60d", "BTI_BritishAmerican_ret_20d", "DHR_vol_20d", "MRK_Merck_zscore_60d", "BTI_BritishAmerican_ret_5d"], "is_new": true}, {"model_id": "new_h5_NORMAL_RandomForest_N8_t7", "algo": "RandomForest", "regime": "NORMAL", "horizon": 5, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "MSTR_Bitcoin3_ret_20d", "CPB_CampbellSoup_zscore_60d", "GILD_Gilead_ret_20d", "EQIX_Equinix_ret_5d", "DOW_Price_zscore_60d", "TED_Spread_vol_20d"], "is_new": true}, {"model_id": "new_h5_NORMAL_RandomForest_N10_t0", "algo": "RandomForest", "regime": "NORMAL", "horizon": 5, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "MSTR_Bitcoin3_ret_5d", "VOD_Vodafone_zscore_60d", "SBUX_vol_20d", "T10Y2Y_Spread_ret_5d", "LLY_zscore_60d", "DAX_Germany_vol_20d", "EWH_HongKong_ret_5d", "NWL_Newell_ret_20d"], "is_new": true}, {"model_id": "new_h5_NORMAL_RandomForest_N10_t1", "algo": "RandomForest", "regime": "NORMAL", "horizon": 5, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "heston_var_ev_h5", "GILD_Gilead_ret_20d", "GD_GeneralDynamics_zscore_60d", "MSTR_Bitcoin3_ret_1d", "EWM_Malaysia_zscore_60d", "Industrial_Production_zscore_60d", "T_ret_1d", "ASX_Australia_vol_20d"], "is_new": true}, {"model_id": "new_h5_NORMAL_RandomForest_N10_t2", "algo": "RandomForest", "regime": "NORMAL", "horizon": 5, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "XLK_Tech_zscore_60d", "CMCSA_ret_1d", "PAYX_Paychex_vol_20d", "CPB_CampbellSoup_vol_20d", "GD_GeneralDynamics_zscore_60d", "DAX_Germany_vol_20d", "CPB_CampbellSoup_zscore_60d", "DE_Deere_vol_20d"], "is_new": true}, {"model_id": "new_h5_NORMAL_RandomForest_N10_t3", "algo": "RandomForest", "regime": "NORMAL", "horizon": 5, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "T10Y2Y_Spread_ret_5d", "SJM_JM_Smucker_ret_5d", "HD_zscore_60d", "BTI_BritishAmerican_ret_5d", "US7Y_Rate_ret_20d", "EWC_Canada_zscore_60d", "MS_MorganStanley_ret_5d", "heston_ev_h3"], "is_new": true}, {"model_id": "new_h5_NORMAL_RandomForest_N10_t4", "algo": "RandomForest", "regime": "NORMAL", "horizon": 5, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "CMCSA_ret_1d", "EWL_Switzerland_zscore_60d", "HangSeng_HK_ret_5d", "US7Y_Rate_ret_20d", "EXC_Exelon_ret_1d", "NWL_Newell_ret_20d", "AORD_AUS_zscore_60d", "PLD_Prologis_ret_5d"], "is_new": true}, {"model_id": "new_h5_NORMAL_RandomForest_N10_t5", "algo": "RandomForest", "regime": "NORMAL", "horizon": 5, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "DAX_Germany_zscore_60d", "CPB_CampbellSoup_zscore_60d", "VOD_Vodafone_zscore_60d", "spx_vol_5d", "EWQ_France_ret_20d", "TXN_vol_20d", "JNJ_ret_1d", "EMR_Emerson_ret_20d"], "is_new": true}, {"model_id": "new_h5_NORMAL_RandomForest_N10_t6", "algo": "RandomForest", "regime": "NORMAL", "horizon": 5, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "BTI_BritishAmerican_ret_20d", "IBEX_Spain_ret_20d", "ES_Evergy_ret_1d", "EWM_Malaysia_ret_1d", "CCI_CrownCastle_vol_20d", "LOW_Lowes_ret_5d", "SBUX_ret_5d", "US6M_Rate_ret_20d"], "is_new": true}, {"model_id": "new_h5_NORMAL_RandomForest_N10_t7", "algo": "RandomForest", "regime": "NORMAL", "horizon": 5, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "XOM_ret_1d", "XOM_ret_20d", "US3Y_Rate_ret_5d", "TGT_Target_zscore_60d", "spx_vol_5d", "SLB_Schlumberger_ret_5d", "spx_momentum_3d", "EWM_Malaysia_vol_20d"], "is_new": true}, {"model_id": "new_h5_NORMAL_RandomForest_N12_t0", "algo": "RandomForest", "regime": "NORMAL", "horizon": 5, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "spx_vol_5d", "ASX_Australia_vol_20d", "INTC_ret_1d", "EWM_Malaysia_vol_20d", "US7Y_Rate_ret_20d", "JNJ_ret_1d", "IBEX_Spain_ret_20d", "EWS_Singapore_ret_5d", "ENB_EnbridgeInc_ret_1d", "SO_SouthernCo_ret_5d"], "is_new": true}, {"model_id": "new_h5_NORMAL_RandomForest_N12_t1", "algo": "RandomForest", "regime": "NORMAL", "horizon": 5, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "VVIX_ret_20d", "vix_acceleration_1d", "US3M_Rate_vol_20d", "EWH_HongKong_ret_5d", "TM_Telephone_ret_1d", "PFE_ret_1d", "T10Y2Y_Spread_ret_5d", "SLB_Schlumberger_ret_5d", "GE_ret_1d", "EWG_Germany_ret_20d"], "is_new": true}, {"model_id": "new_h5_NORMAL_RandomForest_N12_t2", "algo": "RandomForest", "regime": "NORMAL", "horizon": 5, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "US3Y_Rate_ret_5d", "CCI_CrownCastle_vol_20d", "FedFunds_zscore_60d", "TED_Spread_zscore_60d", "PAYX_Paychex_zscore_60d", "NOC_Northrop_ret_20d", "HD_ret_1d", "EQIX_Equinix_ret_5d", "PAYX_Paychex_ret_20d", "EWL_Switzerland_vol_20d"], "is_new": true}, {"model_id": "new_h5_NORMAL_RandomForest_N12_t3", "algo": "RandomForest", "regime": "NORMAL", "horizon": 5, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "CPB_CampbellSoup_ret_5d", "TGT_Target_zscore_60d", "US30Y_Rate_ret_20d", "SBUX_vol_20d", "BLK_BlackRock_zscore_60d", "NVDA_vol_20d", "heston_var_ev_h3", "WTI_Oil_FRED_zscore_60d", "XOM_ret_20d", "TXN_vol_20d"], "is_new": true}, {"model_id": "new_h5_NORMAL_RandomForest_N12_t4", "algo": "RandomForest", "regime": "NORMAL", "horizon": 5, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "BDX_Becton_Dickinson_ret_20d", "CCI_CrownCastle_vol_20d", "Core_CPI_zscore_60d", "MSTR_Bitcoin3_ret_1d", "EOG_EOGResources_vol_20d", "SLB_Schlumberger_ret_5d", "AMT_AmericanTower_ret_1d", "CLX_Clorox_vol_20d", "NOC_Northrop_ret_20d", "TM_Telephone_vol_20d"], "is_new": true}, {"model_id": "new_h5_NORMAL_RandomForest_N12_t5", "algo": "RandomForest", "regime": "NORMAL", "horizon": 5, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "SPY_zscore_60d", "CPB_CampbellSoup_vol_20d", "US5Y_Rate_ret_5d", "PG_ret_20d", "LUV_SouthwestAir_ret_5d", "XOM_ret_20d", "heston_var_ev_h7", "heston_var_ev_h5", "ASX_Australia_ret_5d", "PCAR_PaccarInc_ret_5d"], "is_new": true}, {"model_id": "new_h5_NORMAL_RandomForest_N12_t6", "algo": "RandomForest", "regime": "NORMAL", "horizon": 5, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "Nikkei_Japan_zscore_60d", "NVDA_vol_20d", "LMT_LockheedMartin_ret_1d", "heston_var_ev_h5", "BDX_Becton_Dickinson_ret_20d", "spx_vol_5d", "IYR_US_REIT2_zscore_60d", "NWL_Newell_ret_20d", "LOW_Lowes_ret_5d", "MO_AltriaMG_ret_1d"], "is_new": true}, {"model_id": "new_h5_NORMAL_RandomForest_N12_t7", "algo": "RandomForest", "regime": "NORMAL", "horizon": 5, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "DOW_Price_zscore_60d", "ORCL_vol_20d", "SCHW_Schwab_ret_5d", "CI_Cigna_vol_20d", "NFCI_ret_5d", "BTI_BritishAmerican_ret_5d", "PCAR_PaccarInc_ret_5d", "DE_Deere_vol_20d", "PFE_ret_1d", "PAYX_Paychex_ret_20d"], "is_new": true}, {"model_id": "new_h5_NORMAL_RandomForest_N15_t0", "algo": "RandomForest", "regime": "NORMAL", "horizon": 5, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWH_HongKong_ret_5d", "CMCSA_ret_1d", "XOM_ret_20d", "gjr_condvar_h1", "hmm_p_stress", "Nikkei_Japan_vol_20d", "CPB_CampbellSoup_ret_20d", "Brent_Oil_FRED_ret_20d", "TM_Telephone_vol_20d", "NEE_NextEra_ret_20d", "VRP_ma5", "SPY_zscore_60d", "CTAS_Cintas_vol_20d"], "is_new": true}, {"model_id": "new_h5_NORMAL_RandomForest_N15_t1", "algo": "RandomForest", "regime": "NORMAL", "horizon": 5, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "US5Y_Rate_ret_5d", "EQR_Equity_ret_1d", "US3M_Rate_zscore_60d", "EWQ_France_zscore_60d", "BA_ret_1d", "AMZN_ret_5d", "HUM_Humana_ret_5d", "GILD_Gilead_ret_20d", "XLF_Fin_vol_20d", "DIS_vol_20d", "EFFR_vol_20d", "US6M_Rate_ret_20d", "GD_GeneralDynamics_zscore_60d"], "is_new": true}, {"model_id": "new_h5_NORMAL_RandomForest_N15_t2", "algo": "RandomForest", "regime": "NORMAL", "horizon": 5, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "NEE_NextEra_ret_20d", "NFCI_ret_5d", "DAX_Germany_vol_20d", "Core_CPI_zscore_60d", "TM_Telephone_ret_1d", "MRK_Merck_zscore_60d", "PG_ret_20d", "ASX_Australia_vol_20d", "AXP_Amex_vol_20d", "CMCSA_ret_1d", "AMT_AmericanTower_ret_1d", "US3Y_Rate_ret_5d", "EFFR_vol_20d"], "is_new": true}, {"model_id": "new_h5_NORMAL_RandomForest_N15_t3", "algo": "RandomForest", "regime": "NORMAL", "horizon": 5, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "US1Y_Rate_ret_20d", "ORCL_vol_20d", "M_Macys_vol_20d", "ITT_ITTInc_ret_5d", "LOW_Lowes_ret_20d", "vix_mean_abs_ret_5d", "QQQ_vol_20d", "Core_CPI_zscore_60d", "XLF_Fin_vol_20d", "Industrial_Production_zscore_60d", "IYM_BasicMaterials_ret_20d", "SBUX_ret_5d", "ENB_EnbridgeInc_ret_1d"], "is_new": true}, {"model_id": "new_h5_NORMAL_RandomForest_N15_t4", "algo": "RandomForest", "regime": "NORMAL", "horizon": 5, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "BTI_BritishAmerican_ret_5d", "spx_abs_ret_max_5d", "EWM_Malaysia_vol_20d", "TXN_vol_20d", "NVDA_vol_20d", "INTC_ret_1d", "Brent_Oil_FRED_ret_20d", "PG_ret_20d", "MSTR_Bitcoin3_ret_5d", "SJM_JM_Smucker_ret_1d", "spx_momentum_3d", "heston_var_ev_h5", "hmm_p_stress"], "is_new": true}, {"model_id": "new_h5_NORMAL_RandomForest_N15_t5", "algo": "RandomForest", "regime": "NORMAL", "horizon": 5, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EOG_EOGResources_ret_5d", "LLY_zscore_60d", "VOD_Vodafone_zscore_60d", "AORD_AUS_zscore_60d", "MS_MorganStanley_zscore_60d", "EWA_Australia_ret_1d", "XLK_Tech_zscore_60d", "EWL_Switzerland_zscore_60d", "Core_PCE_zscore_60d", "EFFR_vol_20d", "LOW_Lowes_ret_5d", "Brent_Oil_FRED_ret_5d", "spx_vol_5d"], "is_new": true}, {"model_id": "new_h5_NORMAL_RandomForest_N15_t6", "algo": "RandomForest", "regime": "NORMAL", "horizon": 5, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "XOM_ret_1d", "VOD_Vodafone_zscore_60d", "TED_Spread_vol_20d", "EQR_Equity_ret_1d", "GE_ret_1d", "T10Y2Y_Spread_ret_5d", "Nikkei_Japan_zscore_60d", "CPB_CampbellSoup_ret_20d", "EWL_Switzerland_vol_20d", "CI_Cigna_vol_20d", "3M_ret_5d", "EWA_Australia_ret_1d", "IWM_SmallCap_vol_20d"], "is_new": true}, {"model_id": "new_h5_NORMAL_RandomForest_N15_t7", "algo": "RandomForest", "regime": "NORMAL", "horizon": 5, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "CI_Cigna_vol_20d", "BDX_Becton_Dickinson_ret_20d", "VRP_ma5", "CMCSA_ret_1d", "ENB_EnbridgeInc_ret_1d", "EWY_Korea_zscore_60d", "JNJ_ret_1d", "US6M_Rate_ret_20d", "EWM_Malaysia_vol_20d", "SLB_Schlumberger_ret_5d", "QQQ_vol_20d", "XLF_Fin_vol_20d", "LOW_Lowes_ret_5d"], "is_new": true}, {"model_id": "new_h5_NORMAL_RandomForest_N20_t0", "algo": "RandomForest", "regime": "NORMAL", "horizon": 5, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "ITT_ITTInc_ret_5d", "EWY_Korea_ret_20d", "IWM_SmallCap_vol_20d", "GILD_Gilead_ret_20d", "MS_MorganStanley_ret_5d", "TXN_vol_20d", "Core_CPI_zscore_60d", "XLY_Disc_vol_20d", "NOC_Northrop_ret_20d", "MSTR_Bitcoin3_ret_5d", "EOG_EOGResources_vol_20d", "EWA_Australia_zscore_60d", "AXP_Amex_vol_20d", "AMD_ret_1d", "EWA_Australia_ret_1d", "PAYX_Paychex_ret_20d", "EWM_Malaysia_zscore_60d", "EWC_Canada_zscore_60d"], "is_new": true}, {"model_id": "new_h5_NORMAL_RandomForest_N20_t1", "algo": "RandomForest", "regime": "NORMAL", "horizon": 5, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "SPY_zscore_60d", "BTI_BritishAmerican_ret_5d", "IYM_BasicMaterials_ret_20d", "BTI_BritishAmerican_ret_20d", "TM_Telephone_ret_1d", "AMD_ret_5d", "PCAR_PaccarInc_ret_5d", "SCHW_Schwab_ret_5d", "TM_Telephone_vol_20d", "EWM_Malaysia_ret_1d", "EWJ_Japan_vol_20d", "Nikkei_Japan_vol_20d", "TED_Spread_vol_20d", "ES_Evergy_ret_1d", "NFCI_ret_5d", "HD_zscore_60d", "HangSeng_HK_ret_5d", "MS_MorganStanley_zscore_60d"], "is_new": true}, {"model_id": "new_h5_NORMAL_RandomForest_N20_t2", "algo": "RandomForest", "regime": "NORMAL", "horizon": 5, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "LMT_LockheedMartin_ret_1d", "EWL_Switzerland_zscore_60d", "BTI_BritishAmerican_ret_5d", "BLK_BlackRock_zscore_60d", "AXP_Amex_ret_20d", "DE_Deere_vol_20d", "NFCI_ret_5d", "Nikkei_Japan_vol_20d", "US30Y_Rate_ret_20d", "XLF_Fin_vol_20d", "PAYX_Paychex_ret_20d", "IBEX_Spain_ret_20d", "EQR_Equity_ret_1d", "QQQ_vol_20d", "INTC_ret_5d", "Brent_Oil_FRED_ret_20d", "GE_ret_1d", "T10Y2Y_Spread_ret_5d"], "is_new": true}, {"model_id": "new_h5_NORMAL_RandomForest_N20_t3", "algo": "RandomForest", "regime": "NORMAL", "horizon": 5, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "LMT_LockheedMartin_ret_1d", "DE_Deere_ret_5d", "XLY_Disc_vol_20d", "CPB_CampbellSoup_ret_5d", "Michigan_Sentiment_ret_20d", "LLY_zscore_60d", "EWG_Germany_ret_20d", "XLF_Fin_vol_20d", "SCHW_Schwab_ret_5d", "US3M_Rate_zscore_60d", "EWY_Korea_zscore_60d", "GILD_Gilead_ret_20d", "PAYX_Paychex_zscore_60d", "AMD_ret_1d", "BTI_BritishAmerican_ret_20d", "vix_acceleration_1d", "IYR_US_REIT2_zscore_60d", "US1Y_Rate_ret_20d"], "is_new": true}, {"model_id": "new_h5_NORMAL_RandomForest_N20_t4", "algo": "RandomForest", "regime": "NORMAL", "horizon": 5, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "US5Y_Rate_ret_5d", "EOG_EOGResources_ret_5d", "PAYX_Paychex_ret_20d", "EFFR_vol_20d", "LOW_Lowes_ret_5d", "EXC_Exelon_ret_1d", "IYM_BasicMaterials_ret_20d", "TED_Spread_zscore_60d", "EWA_Australia_ret_1d", "DOW_Price_zscore_60d", "GILD_Gilead_ret_20d", "AORD_AUS_zscore_60d", "DHR_ret_1d", "ASX_Australia_ret_5d", "T10Y2Y_Spread_ret_5d", "US7Y_Rate_ret_20d", "Brent_Oil_FRED_ret_20d", "DE_Deere_vol_20d"], "is_new": true}, {"model_id": "new_h5_NORMAL_RandomForest_N20_t5", "algo": "RandomForest", "regime": "NORMAL", "horizon": 5, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "CPB_CampbellSoup_vol_20d", "INTC_ret_1d", "ASX_Australia_vol_20d", "CPB_CampbellSoup_zscore_60d", "AXP_Amex_vol_20d", "DOW_Price_zscore_60d", "vix_mean_abs_ret_5d", "EWY_Korea_ret_20d", "US3Y_Rate_ret_5d", "HangSeng_HK_ret_1d", "Core_CPI_zscore_60d", "IBEX_Spain_ret_20d", "spx_vol_5d", "DE_Deere_ret_5d", "VOD_Vodafone_zscore_60d", "BDX_Becton_Dickinson_ret_20d", "BLK_BlackRock_zscore_60d", "US3M_Rate_zscore_60d"], "is_new": true}, {"model_id": "new_h5_NORMAL_RandomForest_N20_t6", "algo": "RandomForest", "regime": "NORMAL", "horizon": 5, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "SBUX_ret_5d", "EXC_Exelon_zscore_60d", "XOM_ret_1d", "heston_ev_h3", "CPB_CampbellSoup_ret_20d", "US3M_Rate_zscore_60d", "EOG_EOGResources_vol_20d", "SLB_Schlumberger_ret_5d", "EWG_Germany_ret_20d", "DHR_ret_1d", "HUM_Humana_ret_5d", "US3M_Rate_vol_20d", "MO_AltriaMG_ret_1d", "CTAS_Cintas_vol_20d", "3M_vol_20d", "VRP_ma5", "Nikkei_Japan_vol_20d", "PAYX_Paychex_zscore_60d"], "is_new": true}, {"model_id": "new_h5_NORMAL_RandomForest_N20_t7", "algo": "RandomForest", "regime": "NORMAL", "horizon": 5, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "Brent_Oil_FRED_ret_20d", "NOC_Northrop_ret_20d", "US5Y_Rate_ret_5d", "XLF_Fin_vol_20d", "EWM_Malaysia_ret_1d", "HangSeng_HK_ret_1d", "PFE_ret_1d", "heston_var_ev_h5", "US1Y_Rate_ret_5d", "PAYX_Paychex_zscore_60d", "PG_ret_20d", "DHR_vol_20d", "PPL_PPL_ret_1d", "GE_ret_1d", "MS_MorganStanley_ret_1d", "ORCL_zscore_60d", "EWH_HongKong_ret_5d", "3M_ret_5d"], "is_new": true}, {"model_id": "new_h5_NORMAL_RandomForest_N25_t0", "algo": "RandomForest", "regime": "NORMAL", "horizon": 5, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "SBUX_vol_20d", "US1Y_Rate_ret_20d", "EOG_EOGResources_vol_20d", "Brent_Oil_FRED_ret_20d", "heston_var_ev_h7", "US7Y_Rate_ret_20d", "DE_Deere_vol_20d", "SBUX_ret_5d", "XLB_Materials_zscore_60d", "AORD_AUS_zscore_60d", "BDX_Becton_Dickinson_ret_20d", "Michigan_Sentiment_ret_20d", "SO_SouthernCo_ret_5d", "EWY_Korea_zscore_60d", "MS_MorganStanley_ret_5d", "HD_zscore_60d", "EQIX_Equinix_ret_5d", "TGT_Target_zscore_60d", "SJM_JM_Smucker_ret_1d", "EWY_Korea_ret_20d", "HangSeng_HK_ret_5d", "LUV_SouthwestAir_ret_5d", "PAYX_Paychex_ret_20d"], "is_new": true}, {"model_id": "new_h5_NORMAL_RandomForest_N25_t1", "algo": "RandomForest", "regime": "NORMAL", "horizon": 5, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWA_Australia_zscore_60d", "CPB_CampbellSoup_ret_20d", "CLX_Clorox_vol_20d", "CI_Cigna_vol_20d", "EWL_Switzerland_vol_20d", "LUV_SouthwestAir_ret_5d", "HD_ret_5d", "XLK_Tech_zscore_60d", "EQR_Equity_ret_1d", "INTC_ret_1d", "US30Y_Rate_ret_20d", "MSTR_Bitcoin3_ret_20d", "PPL_PPL_ret_1d", "GILD_Gilead_ret_20d", "INTC_ret_5d", "NEE_NextEra_ret_20d", "HD_zscore_60d", "IBEX_Spain_ret_20d", "EWM_Malaysia_ret_1d", "Core_PCE_zscore_60d", "EWG_Germany_vol_20d", "EMR_Emerson_ret_20d", "TM_Telephone_ret_1d"], "is_new": true}, {"model_id": "new_h5_NORMAL_RandomForest_N25_t2", "algo": "RandomForest", "regime": "NORMAL", "horizon": 5, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWG_Germany_ret_20d", "NVDA_vol_20d", "SJM_JM_Smucker_ret_5d", "ES_Evergy_ret_1d", "EWL_Switzerland_zscore_60d", "EWQ_France_ret_20d", "DIS_vol_20d", "XOM_ret_1d", "ASX_Australia_ret_5d", "HD_zscore_60d", "Nikkei_Japan_zscore_60d", "3M_ret_5d", "SPY_zscore_60d", "heston_ev_h3", "MSTR_Bitcoin3_ret_5d", "PAYX_Paychex_ret_20d", "HangSeng_HK_ret_1d", "heston_var_ev_h3", "PPL_PPL_ret_1d", "DE_Deere_vol_20d", "HD_ret_5d", "TED_Spread_zscore_60d", "AXP_Amex_ret_20d"], "is_new": true}, {"model_id": "new_h5_NORMAL_RandomForest_N25_t3", "algo": "RandomForest", "regime": "NORMAL", "horizon": 5, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "CPB_CampbellSoup_vol_20d", "SBUX_vol_20d", "XLB_Materials_zscore_60d", "AMT_AmericanTower_ret_1d", "TM_Telephone_vol_20d", "EXC_Exelon_ret_1d", "US1Y_Rate_ret_5d", "CCI_CrownCastle_vol_20d", "INTC_ret_5d", "EFFR_vol_20d", "SBUX_zscore_60d", "PAYX_Paychex_zscore_60d", "CPB_CampbellSoup_zscore_60d", "Brent_Oil_FRED_ret_20d", "LOW_Lowes_ret_20d", "EMR_Emerson_ret_20d", "SCHW_Schwab_ret_5d", "ITT_ITTInc_ret_5d", "AMD_ret_5d", "PLD_Prologis_ret_5d", "EWG_Germany_vol_20d", "NWL_Newell_ret_20d", "CI_Cigna_vol_20d"], "is_new": true}, {"model_id": "new_h5_NORMAL_RandomForest_N25_t4", "algo": "RandomForest", "regime": "NORMAL", "horizon": 5, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "BDX_Becton_Dickinson_ret_20d", "MS_MorganStanley_zscore_60d", "EWY_Korea_ret_20d", "EMR_Emerson_ret_20d", "EOG_EOGResources_ret_5d", "EWA_Australia_zscore_60d", "MS_MorganStanley_ret_1d", "EOG_EOGResources_vol_20d", "SLB_Schlumberger_ret_1d", "CPB_CampbellSoup_ret_20d", "US6M_Rate_ret_20d", "PCAR_PaccarInc_ret_5d", "US3M_Rate_vol_20d", "CCI_CrownCastle_vol_20d", "CPB_CampbellSoup_vol_20d", "CI_Cigna_vol_20d", "EWJ_Japan_vol_20d", "AMZN_ret_5d", "QQQ_vol_20d", "EWM_Malaysia_vol_20d", "NVDA_vol_20d", "heston_var_ev_h7", "CMCSA_ret_1d"], "is_new": true}, {"model_id": "new_h5_NORMAL_RandomForest_N25_t5", "algo": "RandomForest", "regime": "NORMAL", "horizon": 5, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "US6M_Rate_ret_20d", "ORCL_zscore_60d", "VOD_Vodafone_zscore_60d", "Nikkei_Japan_zscore_60d", "VVIX_ret_20d", "vix_acceleration_1d", "TED_Spread_vol_20d", "BTI_BritishAmerican_ret_5d", "MSTR_Bitcoin3_ret_20d", "EOG_EOGResources_vol_20d", "M_Macys_vol_20d", "heston_var_ev_h3", "SBUX_zscore_60d", "CPB_CampbellSoup_ret_20d", "GE_ret_1d", "BTI_BritishAmerican_ret_20d", "spx_momentum_3d", "TXN_vol_20d", "heston_ev_h3", "NEE_NextEra_ret_20d", "LUV_SouthwestAir_ret_5d", "vix_mean_abs_ret_5d", "SCHW_Schwab_ret_5d"], "is_new": true}, {"model_id": "new_h5_NORMAL_RandomForest_N25_t6", "algo": "RandomForest", "regime": "NORMAL", "horizon": 5, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EMR_Emerson_ret_20d", "CTAS_Cintas_vol_20d", "DHR_ret_1d", "PAYX_Paychex_zscore_60d", "Brent_Oil_FRED_ret_20d", "spx_vol_5d", "CI_Cigna_vol_20d", "LMT_LockheedMartin_vol_20d", "DAX_Germany_vol_20d", "XLY_Disc_vol_20d", "PAYX_Paychex_ret_20d", "AMGN_Amgen_ret_1d", "XLF_Fin_vol_20d", "Michigan_Sentiment_ret_20d", "EFFR_ret_1d", "PFE_ret_1d", "EWL_Switzerland_zscore_60d", "CPB_CampbellSoup_zscore_60d", "Nikkei_Japan_vol_20d", "QQQ_vol_20d", "EWG_Germany_vol_20d", "EWY_Korea_zscore_60d", "US3M_Rate_zscore_60d"], "is_new": true}, {"model_id": "new_h5_NORMAL_RandomForest_N25_t7", "algo": "RandomForest", "regime": "NORMAL", "horizon": 5, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "Nikkei_Japan_zscore_60d", "SBUX_vol_20d", "T10Y2Y_Spread_ret_5d", "NOC_Northrop_ret_20d", "CMCSA_ret_1d", "PAYX_Paychex_vol_20d", "SJM_JM_Smucker_ret_5d", "HD_ret_1d", "AMD_ret_1d", "ITT_ITTInc_ret_5d", "HangSeng_HK_ret_1d", "TM_Telephone_vol_20d", "LMT_LockheedMartin_ret_1d", "CTAS_Cintas_vol_20d", "DAX_Germany_zscore_60d", "AMGN_Amgen_ret_1d", "NEE_NextEra_ret_20d", "CPB_CampbellSoup_zscore_60d", "XLV_Health_zscore_60d", "MSTR_Bitcoin3_ret_1d", "SO_SouthernCo_ret_5d", "QQQ_vol_20d", "EWY_Korea_ret_20d"], "is_new": true}, {"model_id": "new_h5_NORMAL_RandomForest_N30_t0", "algo": "RandomForest", "regime": "NORMAL", "horizon": 5, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "GD_GeneralDynamics_zscore_60d", "XLY_Disc_vol_20d", "SJM_JM_Smucker_ret_5d", "PCAR_PaccarInc_ret_5d", "SBUX_ret_5d", "CPB_CampbellSoup_ret_5d", "LUV_SouthwestAir_ret_5d", "TED_Spread_vol_20d", "XLF_Fin_vol_20d", "DAX_Germany_zscore_60d", "SCHW_Schwab_ret_5d", "US1Y_Rate_ret_5d", "hmm_p_stress", "DIS_vol_20d", "LOW_Lowes_ret_20d", "VVIX_ret_20d", "SLB_Schlumberger_ret_1d", "EWM_Malaysia_zscore_60d", "EWL_Switzerland_vol_20d", "US3M_Rate_vol_20d", "XLK_Tech_zscore_60d", "Brent_Oil_FRED_ret_5d", "SBUX_zscore_60d", "US6M_Rate_ret_20d", "EWJ_Japan_vol_20d", "NVDA_vol_20d", "spx_vol_5d", "US3Y_Rate_ret_5d"], "is_new": true}, {"model_id": "new_h5_NORMAL_RandomForest_N30_t1", "algo": "RandomForest", "regime": "NORMAL", "horizon": 5, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "BTI_BritishAmerican_ret_5d", "CCI_CrownCastle_vol_20d", "DE_Deere_vol_20d", "MS_MorganStanley_ret_5d", "vix_mean_abs_ret_5d", "3M_vol_20d", "VOD_Vodafone_zscore_60d", "BTI_BritishAmerican_ret_20d", "spx_vol_5d", "TED_Spread_zscore_60d", "DOW_Price_zscore_60d", "Retail_Sales_zscore_60d", "SBUX_vol_20d", "INTC_ret_1d", "MSTR_Bitcoin3_ret_1d", "LMT_LockheedMartin_ret_1d", "EWA_Australia_ret_1d", "US3M_Rate_zscore_60d", "US30Y_Rate_ret_20d", "EWJ_Japan_vol_20d", "SBUX_zscore_60d", "Nikkei_Japan_vol_20d", "PLD_Prologis_ret_5d", "EWM_Malaysia_vol_20d", "EFFR_vol_20d", "NVDA_vol_20d", "HangSeng_HK_vol_20d", "EMR_Emerson_ret_20d"], "is_new": true}, {"model_id": "new_h5_NORMAL_RandomForest_N30_t2", "algo": "RandomForest", "regime": "NORMAL", "horizon": 5, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWJ_Japan_vol_20d", "AORD_AUS_zscore_60d", "CLX_Clorox_vol_20d", "FedFunds_zscore_60d", "DE_Deere_vol_20d", "EWG_Germany_ret_20d", "JNJ_ret_1d", "DHR_ret_1d", "SBUX_vol_20d", "XLV_Health_zscore_60d", "GD_GeneralDynamics_zscore_60d", "EWQ_France_zscore_60d", "HD_ret_20d", "ASX_Australia_vol_20d", "PFE_ret_1d", "Brent_Oil_FRED_ret_20d", "Michigan_Sentiment_ret_20d", "DHR_vol_20d", "heston_var_ev_h3", "M_Macys_vol_20d", "ORCL_vol_20d", "EXC_Exelon_zscore_60d", "SBUX_ret_5d", "TED_Spread_zscore_60d", "HangSeng_HK_vol_20d", "T10Y2Y_Spread_ret_5d", "VRP_ma5", "vix_acceleration_1d"], "is_new": true}, {"model_id": "new_h5_NORMAL_RandomForest_N30_t3", "algo": "RandomForest", "regime": "NORMAL", "horizon": 5, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "Core_PCE_zscore_60d", "EWM_Malaysia_vol_20d", "CCI_CrownCastle_vol_20d", "AMT_AmericanTower_ret_1d", "M_Macys_vol_20d", "spx_abs_ret_max_5d", "MS_MorganStanley_ret_5d", "AXP_Amex_ret_20d", "EFFR_ret_1d", "XLK_Tech_zscore_60d", "SCHW_Schwab_ret_5d", "DHR_vol_20d", "XLV_Health_zscore_60d", "HangSeng_HK_ret_5d", "IYR_US_REIT2_zscore_60d", "vix_mean_abs_ret_5d", "EWG_Germany_ret_20d", "LMT_LockheedMartin_vol_20d", "NFCI_ret_5d", "3M_vol_20d", "VRP_ma5", "BA_ret_1d", "Brent_Oil_FRED_ret_5d", "EXC_Exelon_zscore_60d", "T10Y2Y_Spread_ret_5d", "AMD_ret_5d", "hmm_p_stress", "US1Y_Rate_ret_20d"], "is_new": true}, {"model_id": "new_h5_NORMAL_RandomForest_N30_t4", "algo": "RandomForest", "regime": "NORMAL", "horizon": 5, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "VRP_ma5", "LUV_SouthwestAir_ret_5d", "3M_ret_5d", "EWG_Germany_vol_20d", "heston_var_ev_h5", "M_Macys_vol_20d", "EWL_Switzerland_vol_20d", "DIS_vol_20d", "Nikkei_Japan_zscore_60d", "EOG_EOGResources_ret_5d", "EWM_Malaysia_vol_20d", "QQQ_vol_20d", "ITT_ITTInc_ret_5d", "AMT_AmericanTower_ret_1d", "IYR_US_REIT2_zscore_60d", "HUM_Humana_ret_5d", "EWH_HongKong_ret_5d", "EFFR_ret_1d", "ES_Evergy_ret_1d", "NVDA_vol_20d", "AMD_ret_1d", "EFFR_vol_20d", "SBUX_ret_5d", "US30Y_Rate_ret_20d", "GILD_Gilead_ret_20d", "MS_MorganStanley_ret_1d", "EQR_Equity_ret_1d", "HD_ret_20d"], "is_new": true}, {"model_id": "new_h5_NORMAL_RandomForest_N30_t5", "algo": "RandomForest", "regime": "NORMAL", "horizon": 5, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "IYR_US_REIT2_zscore_60d", "VVIX_ret_20d", "EXC_Exelon_zscore_60d", "LOW_Lowes_ret_20d", "TED_Spread_vol_20d", "SBUX_ret_5d", "BA_ret_1d", "DHR_vol_20d", "EWJ_Japan_vol_20d", "AMGN_Amgen_ret_1d", "PPL_PPL_ret_1d", "BTI_BritishAmerican_ret_20d", "AXP_Amex_ret_20d", "IYM_BasicMaterials_ret_20d", "MSTR_Bitcoin3_ret_1d", "HangSeng_HK_vol_20d", "AVB_AvalonBay_zscore_60d", "AMT_AmericanTower_ret_1d", "NFCI_ret_5d", "PAYX_Paychex_zscore_60d", "XLK_Tech_zscore_60d", "CPB_CampbellSoup_zscore_60d", "MRK_Merck_zscore_60d", "spx_abs_ret_max_5d", "MSTR_Bitcoin3_ret_5d", "SCHW_Schwab_ret_5d", "INTC_ret_5d", "DE_Deere_ret_5d"], "is_new": true}, {"model_id": "new_h5_NORMAL_RandomForest_N30_t6", "algo": "RandomForest", "regime": "NORMAL", "horizon": 5, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "CPB_CampbellSoup_ret_5d", "INTC_ret_1d", "spx_abs_ret_max_5d", "heston_var_ev_h5", "MO_AltriaMG_ret_1d", "AMGN_Amgen_ret_1d", "PCAR_PaccarInc_ret_5d", "IYM_BasicMaterials_ret_20d", "SBUX_zscore_60d", "NOC_Northrop_ret_20d", "EXC_Exelon_zscore_60d", "HangSeng_HK_vol_20d", "US5Y_Rate_ret_5d", "MSTR_Bitcoin3_ret_1d", "EWY_Korea_ret_20d", "US3Y_Rate_ret_5d", "EQR_Equity_ret_1d", "CPB_CampbellSoup_ret_20d", "HangSeng_HK_ret_5d", "US1Y_Rate_ret_5d", "Core_PCE_zscore_60d", "US3M_Rate_zscore_60d", "MS_MorganStanley_ret_5d", "IBEX_Spain_ret_20d", "ENB_EnbridgeInc_ret_1d", "GILD_Gilead_ret_20d", "BLK_BlackRock_zscore_60d", "HD_ret_20d"], "is_new": true}, {"model_id": "new_h5_NORMAL_RandomForest_N30_t7", "algo": "RandomForest", "regime": "NORMAL", "horizon": 5, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "ITT_ITTInc_ret_5d", "CCI_CrownCastle_vol_20d", "VRP_ma5", "Brent_Oil_FRED_ret_5d", "Nikkei_Japan_vol_20d", "AXP_Amex_vol_20d", "hmm_p_stress", "heston_var_ev_h3", "EQIX_Equinix_ret_5d", "DHR_ret_1d", "DOW_Price_zscore_60d", "spx_abs_ret_max_5d", "IYM_BasicMaterials_ret_20d", "NWL_Newell_ret_20d", "SCHW_Schwab_ret_5d", "VVIX_ret_20d", "EFFR_vol_20d", "NVDA_vol_20d", "MSTR_Bitcoin3_ret_1d", "DE_Deere_vol_20d", "HangSeng_HK_ret_5d", "EWM_Malaysia_vol_20d", "HD_ret_1d", "CPB_CampbellSoup_ret_20d", "PG_ret_20d", "TXN_vol_20d", "BTI_BritishAmerican_ret_20d", "spx_vol_5d"], "is_new": true}, {"model_id": "new_h5_NORMAL_LogisticRegression_N5_t0", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 5, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "hmm_p_stress", "US1Y_Rate_ret_5d", "MSTR_Bitcoin3_ret_20d"], "is_new": true}, {"model_id": "new_h5_NORMAL_LogisticRegression_N5_t1", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 5, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "spx_momentum_3d", "Core_CPI_zscore_60d", "Brent_Oil_FRED_ret_5d"], "is_new": true}, {"model_id": "new_h5_NORMAL_LogisticRegression_N5_t2", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 5, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "LOW_Lowes_ret_5d", "Retail_Sales_zscore_60d", "GILD_Gilead_ret_20d"], "is_new": true}, {"model_id": "new_h5_NORMAL_LogisticRegression_N5_t3", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 5, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "Nikkei_Japan_vol_20d", "SBUX_vol_20d", "AMD_ret_5d"], "is_new": true}, {"model_id": "new_h5_NORMAL_LogisticRegression_N5_t4", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 5, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "QQQ_vol_20d", "Brent_Oil_FRED_ret_20d", "TM_Telephone_ret_1d"], "is_new": true}, {"model_id": "new_h5_NORMAL_LogisticRegression_N5_t5", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 5, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "XLB_Materials_zscore_60d", "BA_ret_1d", "EXC_Exelon_ret_1d"], "is_new": true}, {"model_id": "new_h5_NORMAL_LogisticRegression_N5_t6", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 5, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "T10Y2Y_Spread_ret_5d", "EWM_Malaysia_zscore_60d", "INTC_ret_5d"], "is_new": true}, {"model_id": "new_h5_NORMAL_LogisticRegression_N5_t7", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 5, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "HD_ret_20d", "ES_Evergy_ret_1d", "PLD_Prologis_ret_5d"], "is_new": true}, {"model_id": "new_h5_NORMAL_LogisticRegression_N8_t0", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 5, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EQR_Equity_ret_1d", "BTI_BritishAmerican_ret_20d", "DOW_Price_zscore_60d", "IYR_US_REIT2_zscore_60d", "spx_abs_ret_max_5d", "US7Y_Rate_ret_20d"], "is_new": true}, {"model_id": "new_h5_NORMAL_LogisticRegression_N8_t1", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 5, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "XOM_ret_20d", "DE_Deere_vol_20d", "HD_ret_5d", "Core_PCE_zscore_60d", "ENB_EnbridgeInc_ret_1d", "NWL_Newell_ret_20d"], "is_new": true}, {"model_id": "new_h5_NORMAL_LogisticRegression_N8_t2", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 5, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "NWL_Newell_ret_20d", "US30Y_Rate_ret_20d", "IWM_SmallCap_vol_20d", "MS_MorganStanley_ret_5d", "NFCI_ret_5d", "AXP_Amex_ret_20d"], "is_new": true}, {"model_id": "new_h5_NORMAL_LogisticRegression_N8_t3", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 5, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "HangSeng_HK_ret_5d", "BTI_BritishAmerican_ret_5d", "Brent_Oil_FRED_ret_5d", "Core_PCE_zscore_60d", "heston_ev_h3", "DE_Deere_ret_5d"], "is_new": true}, {"model_id": "new_h5_NORMAL_LogisticRegression_N8_t4", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 5, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EFFR_ret_1d", "MSTR_Bitcoin3_ret_1d", "CMCSA_ret_1d", "IYR_US_REIT2_zscore_60d", "MO_AltriaMG_ret_1d", "EXC_Exelon_ret_1d"], "is_new": true}, {"model_id": "new_h5_NORMAL_LogisticRegression_N8_t5", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 5, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "vix_mean_abs_ret_5d", "CMCSA_ret_1d", "heston_var_ev_h7", "GE_ret_1d", "BTI_BritishAmerican_ret_20d", "BA_ret_1d"], "is_new": true}, {"model_id": "new_h5_NORMAL_LogisticRegression_N8_t6", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 5, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "NEE_NextEra_ret_20d", "HD_zscore_60d", "Nikkei_Japan_zscore_60d", "SLB_Schlumberger_ret_1d", "SLB_Schlumberger_ret_5d", "AMT_AmericanTower_ret_1d"], "is_new": true}, {"model_id": "new_h5_NORMAL_LogisticRegression_N8_t7", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 5, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EFFR_ret_1d", "MSTR_Bitcoin3_ret_20d", "AORD_AUS_zscore_60d", "IYR_US_REIT2_zscore_60d", "CPB_CampbellSoup_zscore_60d", "SJM_JM_Smucker_ret_5d"], "is_new": true}, {"model_id": "new_h5_NORMAL_LogisticRegression_N10_t0", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 5, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "CPB_CampbellSoup_ret_20d", "EWG_Germany_ret_20d", "SBUX_vol_20d", "DIS_vol_20d", "US30Y_Rate_ret_20d", "CPB_CampbellSoup_zscore_60d", "DAX_Germany_zscore_60d", "ENB_EnbridgeInc_ret_1d"], "is_new": true}, {"model_id": "new_h5_NORMAL_LogisticRegression_N10_t1", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 5, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "US1Y_Rate_ret_5d", "TGT_Target_zscore_60d", "Michigan_Sentiment_ret_20d", "EWM_Malaysia_vol_20d", "NOC_Northrop_ret_20d", "NWL_Newell_ret_20d", "SLB_Schlumberger_ret_1d", "ASX_Australia_ret_5d"], "is_new": true}, {"model_id": "new_h5_NORMAL_LogisticRegression_N10_t2", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 5, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "TXN_vol_20d", "JNJ_ret_1d", "US3M_Rate_vol_20d", "heston_ev_h3", "VVIX_ret_20d", "PAYX_Paychex_vol_20d", "IYM_BasicMaterials_ret_20d", "ES_Evergy_ret_1d"], "is_new": true}, {"model_id": "new_h5_NORMAL_LogisticRegression_N10_t3", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 5, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "BDX_Becton_Dickinson_ret_20d", "LLY_zscore_60d", "NFCI_ret_5d", "DHR_ret_1d", "TED_Spread_vol_20d", "HD_zscore_60d", "HD_ret_20d", "EWA_Australia_zscore_60d"], "is_new": true}, {"model_id": "new_h5_NORMAL_LogisticRegression_N10_t4", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 5, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "Nikkei_Japan_vol_20d", "AXP_Amex_ret_20d", "VVIX_ret_20d", "NFCI_ret_5d", "EQIX_Equinix_ret_5d", "XLV_Health_zscore_60d", "EWY_Korea_ret_20d", "US3M_Rate_vol_20d"], "is_new": true}, {"model_id": "new_h5_NORMAL_LogisticRegression_N10_t5", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 5, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWA_Australia_zscore_60d", "MSTR_Bitcoin3_ret_20d", "CMCSA_ret_1d", "IWM_SmallCap_vol_20d", "CPB_CampbellSoup_vol_20d", "AMD_ret_1d", "3M_vol_20d", "EWG_Germany_vol_20d"], "is_new": true}, {"model_id": "new_h5_NORMAL_LogisticRegression_N10_t6", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 5, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "DOW_Price_zscore_60d", "AMD_ret_1d", "XLY_Disc_vol_20d", "NVDA_vol_20d", "XLV_Health_zscore_60d", "SO_SouthernCo_ret_5d", "MSTR_Bitcoin3_ret_5d", "ASX_Australia_vol_20d"], "is_new": true}, {"model_id": "new_h5_NORMAL_LogisticRegression_N10_t7", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 5, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "PAYX_Paychex_vol_20d", "CMCSA_ret_1d", "LOW_Lowes_ret_5d", "IYM_BasicMaterials_ret_20d", "MSTR_Bitcoin3_ret_5d", "DHR_ret_1d", "EWA_Australia_ret_1d", "US5Y_Rate_ret_5d"], "is_new": true}, {"model_id": "new_h5_NORMAL_LogisticRegression_N12_t0", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 5, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "IBEX_Spain_ret_20d", "NEE_NextEra_ret_20d", "XLV_Health_zscore_60d", "BTI_BritishAmerican_ret_20d", "HD_ret_20d", "Industrial_Production_zscore_60d", "PCAR_PaccarInc_ret_5d", "SLB_Schlumberger_ret_5d", "MRK_Merck_zscore_60d", "XLY_Disc_vol_20d"], "is_new": true}, {"model_id": "new_h5_NORMAL_LogisticRegression_N12_t1", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 5, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "XLB_Materials_zscore_60d", "T_ret_1d", "AORD_AUS_zscore_60d", "VOD_Vodafone_zscore_60d", "INTC_ret_1d", "VVIX_ret_20d", "XLY_Disc_vol_20d", "BLK_BlackRock_zscore_60d", "LOW_Lowes_ret_20d", "MSTR_Bitcoin3_ret_1d"], "is_new": true}, {"model_id": "new_h5_NORMAL_LogisticRegression_N12_t2", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 5, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "Michigan_Sentiment_ret_20d", "DE_Deere_vol_20d", "JNJ_ret_1d", "CPB_CampbellSoup_ret_20d", "BDX_Becton_Dickinson_ret_20d", "EWM_Malaysia_zscore_60d", "LUV_SouthwestAir_ret_5d", "EWY_Korea_ret_20d", "AMZN_ret_5d", "SLB_Schlumberger_ret_5d"], "is_new": true}, {"model_id": "new_h5_NORMAL_LogisticRegression_N12_t3", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 5, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "3M_ret_5d", "EMR_Emerson_ret_20d", "GILD_Gilead_ret_20d", "NFCI_ret_5d", "SBUX_ret_5d", "Retail_Sales_zscore_60d", "NVDA_vol_20d", "vix_acceleration_1d", "XOM_ret_1d", "US1Y_Rate_ret_20d"], "is_new": true}, {"model_id": "new_h5_NORMAL_LogisticRegression_N12_t4", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 5, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "HUM_Humana_ret_5d", "CLX_Clorox_vol_20d", "BDX_Becton_Dickinson_ret_20d", "SLB_Schlumberger_ret_5d", "BA_ret_1d", "XOM_ret_1d", "spx_vol_5d", "BTI_BritishAmerican_ret_20d", "3M_vol_20d", "PPL_PPL_ret_1d"], "is_new": true}, {"model_id": "new_h5_NORMAL_LogisticRegression_N12_t5", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 5, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EOG_EOGResources_ret_5d", "EQIX_Equinix_ret_5d", "HD_ret_20d", "TED_Spread_vol_20d", "Industrial_Production_zscore_60d", "LMT_LockheedMartin_ret_1d", "Core_CPI_zscore_60d", "JNJ_ret_1d", "gjr_condvar_h1", "PAYX_Paychex_vol_20d"], "is_new": true}, {"model_id": "new_h5_NORMAL_LogisticRegression_N12_t6", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 5, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "HD_ret_20d", "US3Y_Rate_ret_5d", "CMCSA_ret_1d", "ORCL_zscore_60d", "Brent_Oil_FRED_ret_5d", "spx_vol_5d", "AXP_Amex_vol_20d", "BA_ret_1d", "EWM_Malaysia_zscore_60d", "spx_abs_ret_max_5d"], "is_new": true}, {"model_id": "new_h5_NORMAL_LogisticRegression_N12_t7", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 5, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "IYR_US_REIT2_zscore_60d", "TGT_Target_zscore_60d", "EWM_Malaysia_vol_20d", "spx_abs_ret_max_5d", "EWL_Switzerland_zscore_60d", "heston_var_ev_h7", "HangSeng_HK_vol_20d", "VRP_ma5", "HD_ret_1d", "EWS_Singapore_ret_5d"], "is_new": true}, {"model_id": "new_h5_NORMAL_LogisticRegression_N15_t0", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 5, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "MRK_Merck_zscore_60d", "BLK_BlackRock_zscore_60d", "Nikkei_Japan_vol_20d", "EWH_HongKong_ret_5d", "M_Macys_vol_20d", "US1Y_Rate_ret_5d", "Industrial_Production_zscore_60d", "US3M_Rate_vol_20d", "XOM_ret_1d", "ES_Evergy_ret_1d", "SLB_Schlumberger_ret_1d", "EWG_Germany_vol_20d", "MSTR_Bitcoin3_ret_1d"], "is_new": true}, {"model_id": "new_h5_NORMAL_LogisticRegression_N15_t1", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 5, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "BTI_BritishAmerican_ret_20d", "LUV_SouthwestAir_ret_5d", "DHR_ret_1d", "LOW_Lowes_ret_20d", "NVDA_vol_20d", "GD_GeneralDynamics_zscore_60d", "CPB_CampbellSoup_ret_5d", "hmm_p_stress", "M_Macys_vol_20d", "LMT_LockheedMartin_vol_20d", "AXP_Amex_vol_20d", "Nikkei_Japan_zscore_60d", "HUM_Humana_ret_5d"], "is_new": true}, {"model_id": "new_h5_NORMAL_LogisticRegression_N15_t2", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 5, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "HUM_Humana_ret_5d", "BLK_BlackRock_zscore_60d", "Michigan_Sentiment_ret_20d", "US3M_Rate_zscore_60d", "NWL_Newell_ret_20d", "MO_AltriaMG_ret_1d", "CPB_CampbellSoup_ret_5d", "GILD_Gilead_ret_20d", "SLB_Schlumberger_ret_5d", "DHR_vol_20d", "MSTR_Bitcoin3_ret_5d", "US1Y_Rate_ret_5d", "EQIX_Equinix_ret_5d"], "is_new": true}, {"model_id": "new_h5_NORMAL_LogisticRegression_N15_t3", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 5, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWY_Korea_ret_20d", "XOM_ret_1d", "ASX_Australia_vol_20d", "BLK_BlackRock_zscore_60d", "US6M_Rate_ret_20d", "heston_ev_h3", "DIS_vol_20d", "EFFR_ret_1d", "HD_ret_1d", "Brent_Oil_FRED_ret_20d", "CMCSA_ret_1d", "QQQ_vol_20d", "Core_PCE_zscore_60d"], "is_new": true}, {"model_id": "new_h5_NORMAL_LogisticRegression_N15_t4", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 5, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "SBUX_zscore_60d", "EWQ_France_zscore_60d", "CTAS_Cintas_vol_20d", "SBUX_ret_5d", "HD_zscore_60d", "EFFR_vol_20d", "BTI_BritishAmerican_ret_20d", "CPB_CampbellSoup_vol_20d", "JNJ_ret_1d", "DE_Deere_ret_5d", "US6M_Rate_ret_20d", "XOM_ret_20d", "Nikkei_Japan_zscore_60d"], "is_new": true}, {"model_id": "new_h5_NORMAL_LogisticRegression_N15_t5", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 5, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "US5Y_Rate_ret_5d", "VOD_Vodafone_zscore_60d", "SCHW_Schwab_ret_5d", "AVB_AvalonBay_zscore_60d", "PPL_PPL_ret_1d", "EWA_Australia_ret_1d", "XLV_Health_zscore_60d", "XLY_Disc_vol_20d", "MO_AltriaMG_ret_1d", "LUV_SouthwestAir_ret_5d", "CPB_CampbellSoup_ret_5d", "EWM_Malaysia_zscore_60d", "CMCSA_ret_1d"], "is_new": true}, {"model_id": "new_h5_NORMAL_LogisticRegression_N15_t6", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 5, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWA_Australia_zscore_60d", "IYR_US_REIT2_zscore_60d", "VVIX_ret_20d", "HUM_Humana_ret_5d", "CPB_CampbellSoup_ret_5d", "DHR_vol_20d", "EWM_Malaysia_ret_1d", "AMD_ret_1d", "vix_mean_abs_ret_5d", "PPL_PPL_ret_1d", "LOW_Lowes_ret_20d", "MSTR_Bitcoin3_ret_1d", "PAYX_Paychex_zscore_60d"], "is_new": true}, {"model_id": "new_h5_NORMAL_LogisticRegression_N15_t7", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 5, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "SBUX_zscore_60d", "ASX_Australia_vol_20d", "XLF_Fin_vol_20d", "MS_MorganStanley_zscore_60d", "EWY_Korea_zscore_60d", "EFFR_ret_1d", "DE_Deere_vol_20d", "LUV_SouthwestAir_ret_5d", "NFCI_ret_5d", "EWH_HongKong_ret_5d", "SLB_Schlumberger_ret_5d", "BTI_BritishAmerican_ret_5d", "ORCL_vol_20d"], "is_new": true}, {"model_id": "new_h5_NORMAL_LogisticRegression_N20_t0", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 5, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "BTI_BritishAmerican_ret_20d", "XLY_Disc_vol_20d", "CCI_CrownCastle_vol_20d", "HangSeng_HK_ret_1d", "FedFunds_zscore_60d", "DE_Deere_ret_5d", "EWA_Australia_zscore_60d", "TM_Telephone_vol_20d", "IBEX_Spain_ret_20d", "IYM_BasicMaterials_ret_20d", "GILD_Gilead_ret_20d", "BDX_Becton_Dickinson_ret_20d", "EWS_Singapore_ret_5d", "DHR_ret_1d", "Retail_Sales_zscore_60d", "GE_ret_1d", "CPB_CampbellSoup_vol_20d", "JNJ_ret_1d"], "is_new": true}, {"model_id": "new_h5_NORMAL_LogisticRegression_N20_t1", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 5, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EFFR_ret_1d", "LUV_SouthwestAir_ret_5d", "MO_AltriaMG_ret_1d", "US3M_Rate_zscore_60d", "DHR_ret_1d", "DE_Deere_vol_20d", "SJM_JM_Smucker_ret_1d", "Brent_Oil_FRED_ret_20d", "XOM_ret_1d", "EQR_Equity_ret_1d", "gjr_condvar_h1", "AMD_ret_1d", "PPL_PPL_ret_1d", "HangSeng_HK_ret_5d", "GILD_Gilead_ret_20d", "CCI_CrownCastle_vol_20d", "vix_mean_abs_ret_5d", "CPB_CampbellSoup_zscore_60d"], "is_new": true}, {"model_id": "new_h5_NORMAL_LogisticRegression_N20_t2", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 5, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "SLB_Schlumberger_ret_5d", "EWA_Australia_ret_1d", "DE_Deere_ret_5d", "heston_var_ev_h3", "SLB_Schlumberger_ret_1d", "HD_ret_5d", "ITT_ITTInc_ret_5d", "EWA_Australia_zscore_60d", "T10Y2Y_Spread_ret_5d", "Core_CPI_zscore_60d", "ENB_EnbridgeInc_ret_1d", "US3Y_Rate_ret_5d", "DAX_Germany_vol_20d", "XOM_ret_20d", "ORCL_vol_20d", "BDX_Becton_Dickinson_ret_20d", "GD_GeneralDynamics_zscore_60d", "AMD_ret_5d"], "is_new": true}, {"model_id": "new_h5_NORMAL_LogisticRegression_N20_t3", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 5, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "TM_Telephone_vol_20d", "PAYX_Paychex_vol_20d", "AVB_AvalonBay_zscore_60d", "BTI_BritishAmerican_ret_20d", "PPL_PPL_ret_1d", "DIS_vol_20d", "Brent_Oil_FRED_ret_5d", "MO_AltriaMG_ret_1d", "CLX_Clorox_vol_20d", "TGT_Target_zscore_60d", "FedFunds_zscore_60d", "heston_var_ev_h7", "PLD_Prologis_ret_5d", "HD_ret_20d", "EXC_Exelon_zscore_60d", "BLK_BlackRock_zscore_60d", "INTC_ret_5d", "EWY_Korea_zscore_60d"], "is_new": true}, {"model_id": "new_h5_NORMAL_LogisticRegression_N20_t4", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 5, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "DAX_Germany_zscore_60d", "EMR_Emerson_ret_20d", "NOC_Northrop_ret_20d", "HangSeng_HK_ret_1d", "US7Y_Rate_ret_20d", "ES_Evergy_ret_1d", "EWL_Switzerland_zscore_60d", "EFFR_ret_1d", "PAYX_Paychex_ret_20d", "ENB_EnbridgeInc_ret_1d", "heston_ev_h3", "HangSeng_HK_vol_20d", "vix_mean_abs_ret_5d", "LOW_Lowes_ret_20d", "ASX_Australia_ret_5d", "SJM_JM_Smucker_ret_5d", "US5Y_Rate_ret_5d", "EWH_HongKong_ret_5d"], "is_new": true}, {"model_id": "new_h5_NORMAL_LogisticRegression_N20_t5", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 5, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "XOM_ret_1d", "US7Y_Rate_ret_20d", "ITT_ITTInc_ret_5d", "EQR_Equity_ret_1d", "EWA_Australia_ret_1d", "DOW_Price_zscore_60d", "INTC_ret_5d", "EWG_Germany_ret_20d", "QQQ_vol_20d", "heston_var_ev_h7", "HUM_Humana_ret_5d", "EFFR_vol_20d", "heston_var_ev_h3", "NFCI_ret_5d", "PAYX_Paychex_ret_20d", "BTI_BritishAmerican_ret_20d", "MSTR_Bitcoin3_ret_20d", "HD_ret_5d"], "is_new": true}, {"model_id": "new_h5_NORMAL_LogisticRegression_N20_t6", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 5, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "DAX_Germany_vol_20d", "HD_ret_5d", "LOW_Lowes_ret_20d", "Michigan_Sentiment_ret_20d", "DE_Deere_vol_20d", "MS_MorganStanley_ret_1d", "AVB_AvalonBay_zscore_60d", "ITT_ITTInc_ret_5d", "JNJ_ret_1d", "EWM_Malaysia_vol_20d", "SJM_JM_Smucker_ret_5d", "LOW_Lowes_ret_5d", "LMT_LockheedMartin_vol_20d", "LMT_LockheedMartin_ret_1d", "TM_Telephone_vol_20d", "Industrial_Production_zscore_60d", "US3Y_Rate_ret_5d", "VOD_Vodafone_zscore_60d"], "is_new": true}, {"model_id": "new_h5_NORMAL_LogisticRegression_N20_t7", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 5, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "PPL_PPL_ret_1d", "PFE_ret_1d", "EWY_Korea_zscore_60d", "GILD_Gilead_ret_20d", "XLV_Health_zscore_60d", "US5Y_Rate_ret_5d", "LMT_LockheedMartin_ret_1d", "US1Y_Rate_ret_5d", "IYR_US_REIT2_zscore_60d", "EWJ_Japan_vol_20d", "XLF_Fin_vol_20d", "CPB_CampbellSoup_vol_20d", "DAX_Germany_vol_20d", "BTI_BritishAmerican_ret_5d", "EWM_Malaysia_ret_1d", "BA_ret_1d", "TGT_Target_zscore_60d", "EWH_HongKong_ret_5d"], "is_new": true}, {"model_id": "new_h5_NORMAL_LogisticRegression_N25_t0", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 5, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "Nikkei_Japan_vol_20d", "CPB_CampbellSoup_ret_5d", "heston_var_ev_h7", "PAYX_Paychex_zscore_60d", "NVDA_vol_20d", "DHR_ret_1d", "LOW_Lowes_ret_20d", "CMCSA_ret_1d", "US5Y_Rate_ret_5d", "PCAR_PaccarInc_ret_5d", "EWG_Germany_vol_20d", "vix_acceleration_1d", "AORD_AUS_zscore_60d", "DIS_vol_20d", "WTI_Oil_FRED_zscore_60d", "SPY_zscore_60d", "EWY_Korea_ret_20d", "EWL_Switzerland_vol_20d", "CPB_CampbellSoup_ret_20d", "DHR_vol_20d", "ORCL_vol_20d", "IWM_SmallCap_vol_20d", "VVIX_ret_20d"], "is_new": true}, {"model_id": "new_h5_NORMAL_LogisticRegression_N25_t1", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 5, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "PAYX_Paychex_ret_20d", "HD_ret_20d", "XLK_Tech_zscore_60d", "TXN_vol_20d", "T_ret_1d", "TM_Telephone_ret_1d", "Retail_Sales_zscore_60d", "Brent_Oil_FRED_ret_5d", "AMD_ret_1d", "LOW_Lowes_ret_20d", "EXC_Exelon_ret_1d", "T10Y2Y_Spread_ret_5d", "HD_zscore_60d", "ASX_Australia_vol_20d", "GILD_Gilead_ret_20d", "EWS_Singapore_ret_5d", "US1Y_Rate_ret_5d", "BA_ret_1d", "BDX_Becton_Dickinson_ret_20d", "TM_Telephone_vol_20d", "MSTR_Bitcoin3_ret_20d", "NEE_NextEra_ret_20d", "NFCI_ret_5d"], "is_new": true}, {"model_id": "new_h5_NORMAL_LogisticRegression_N25_t2", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 5, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "XOM_ret_1d", "EWY_Korea_ret_20d", "MSTR_Bitcoin3_ret_1d", "EFFR_ret_1d", "DIS_vol_20d", "EWQ_France_zscore_60d", "hmm_p_stress", "XLV_Health_zscore_60d", "EWH_HongKong_ret_5d", "MSTR_Bitcoin3_ret_5d", "GILD_Gilead_ret_20d", "Nikkei_Japan_vol_20d", "US3Y_Rate_ret_5d", "LMT_LockheedMartin_ret_1d", "GD_GeneralDynamics_zscore_60d", "US3M_Rate_zscore_60d", "IBEX_Spain_ret_20d", "Nikkei_Japan_zscore_60d", "LOW_Lowes_ret_5d", "US3M_Rate_vol_20d", "DHR_vol_20d", "AMD_ret_5d", "WTI_Oil_FRED_zscore_60d"], "is_new": true}, {"model_id": "new_h5_NORMAL_LogisticRegression_N25_t3", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 5, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWJ_Japan_vol_20d", "DOW_Price_zscore_60d", "MO_AltriaMG_ret_1d", "PG_ret_20d", "WTI_Oil_FRED_zscore_60d", "CLX_Clorox_vol_20d", "NEE_NextEra_ret_20d", "ES_Evergy_ret_1d", "XLY_Disc_vol_20d", "BDX_Becton_Dickinson_ret_20d", "SBUX_ret_5d", "Retail_Sales_zscore_60d", "ENB_EnbridgeInc_ret_1d", "NOC_Northrop_ret_20d", "XLF_Fin_vol_20d", "DAX_Germany_zscore_60d", "LUV_SouthwestAir_ret_5d", "heston_var_ev_h5", "EWH_HongKong_ret_5d", "DE_Deere_ret_5d", "AVB_AvalonBay_zscore_60d", "Michigan_Sentiment_ret_20d", "heston_var_ev_h3"], "is_new": true}, {"model_id": "new_h5_NORMAL_LogisticRegression_N25_t4", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 5, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "ASX_Australia_ret_5d", "SBUX_vol_20d", "AMD_ret_1d", "BTI_BritishAmerican_ret_5d", "DE_Deere_vol_20d", "ASX_Australia_vol_20d", "AMD_ret_5d", "T_ret_1d", "HD_ret_20d", "AXP_Amex_ret_20d", "QQQ_vol_20d", "LUV_SouthwestAir_ret_5d", "SPY_zscore_60d", "SO_SouthernCo_ret_5d", "EMR_Emerson_ret_20d", "DE_Deere_ret_5d", "DIS_vol_20d", "Retail_Sales_zscore_60d", "DAX_Germany_zscore_60d", "IBEX_Spain_ret_20d", "XLY_Disc_vol_20d", "ITT_ITTInc_ret_5d", "EWM_Malaysia_ret_1d"], "is_new": true}, {"model_id": "new_h5_NORMAL_LogisticRegression_N25_t5", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 5, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "SLB_Schlumberger_ret_1d", "spx_vol_5d", "SBUX_ret_5d", "EWG_Germany_vol_20d", "GE_ret_1d", "BA_ret_1d", "LMT_LockheedMartin_ret_1d", "EWM_Malaysia_zscore_60d", "TM_Telephone_vol_20d", "MS_MorganStanley_zscore_60d", "Retail_Sales_zscore_60d", "Michigan_Sentiment_ret_20d", "ORCL_zscore_60d", "PCAR_PaccarInc_ret_5d", "AMD_ret_5d", "EWA_Australia_ret_1d", "vix_acceleration_1d", "XLK_Tech_zscore_60d", "PAYX_Paychex_ret_20d", "US3M_Rate_vol_20d", "XOM_ret_20d", "QQQ_vol_20d", "DHR_vol_20d"], "is_new": true}, {"model_id": "new_h5_NORMAL_LogisticRegression_N25_t6", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 5, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "PAYX_Paychex_ret_20d", "DAX_Germany_vol_20d", "US5Y_Rate_ret_5d", "spx_vol_5d", "spx_momentum_3d", "EWY_Korea_zscore_60d", "Michigan_Sentiment_ret_20d", "MO_AltriaMG_ret_1d", "ASX_Australia_ret_5d", "CLX_Clorox_vol_20d", "ES_Evergy_ret_1d", "EWA_Australia_zscore_60d", "Industrial_Production_zscore_60d", "SBUX_ret_5d", "TED_Spread_vol_20d", "DIS_vol_20d", "EOG_EOGResources_vol_20d", "EWC_Canada_zscore_60d", "GE_ret_1d", "PG_ret_20d", "ASX_Australia_vol_20d", "TM_Telephone_vol_20d", "EWL_Switzerland_zscore_60d"], "is_new": true}, {"model_id": "new_h5_NORMAL_LogisticRegression_N25_t7", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 5, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "MSTR_Bitcoin3_ret_1d", "DHR_vol_20d", "XOM_ret_1d", "MO_AltriaMG_ret_1d", "EWM_Malaysia_zscore_60d", "MS_MorganStanley_zscore_60d", "IBEX_Spain_ret_20d", "heston_var_ev_h7", "Brent_Oil_FRED_ret_20d", "GE_ret_1d", "TM_Telephone_ret_1d", "QQQ_vol_20d", "SBUX_zscore_60d", "ORCL_zscore_60d", "HUM_Humana_ret_5d", "Industrial_Production_zscore_60d", "US3M_Rate_zscore_60d", "US3M_Rate_vol_20d", "GILD_Gilead_ret_20d", "DAX_Germany_vol_20d", "SJM_JM_Smucker_ret_5d", "gjr_condvar_h1", "EWM_Malaysia_vol_20d"], "is_new": true}, {"model_id": "new_h5_NORMAL_LogisticRegression_N30_t0", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 5, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "ENB_EnbridgeInc_ret_1d", "AMT_AmericanTower_ret_1d", "ORCL_zscore_60d", "WTI_Oil_FRED_zscore_60d", "MS_MorganStanley_ret_1d", "HD_ret_20d", "HD_zscore_60d", "TED_Spread_vol_20d", "BTI_BritishAmerican_ret_20d", "QQQ_vol_20d", "LMT_LockheedMartin_vol_20d", "gjr_condvar_h1", "DE_Deere_ret_5d", "IYR_US_REIT2_zscore_60d", "EWM_Malaysia_vol_20d", "spx_momentum_3d", "heston_var_ev_h3", "EQR_Equity_ret_1d", "LMT_LockheedMartin_ret_1d", "ITT_ITTInc_ret_5d", "DHR_ret_1d", "VVIX_ret_20d", "AORD_AUS_zscore_60d", "US7Y_Rate_ret_20d", "EWA_Australia_zscore_60d", "BA_ret_1d", "TXN_vol_20d", "PLD_Prologis_ret_5d"], "is_new": true}, {"model_id": "new_h5_NORMAL_LogisticRegression_N30_t1", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 5, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "spx_vol_5d", "EOG_EOGResources_vol_20d", "PPL_PPL_ret_1d", "SPY_zscore_60d", "QQQ_vol_20d", "BDX_Becton_Dickinson_ret_20d", "EWC_Canada_zscore_60d", "EWY_Korea_ret_20d", "DE_Deere_vol_20d", "Core_CPI_zscore_60d", "Industrial_Production_zscore_60d", "SJM_JM_Smucker_ret_5d", "EWH_HongKong_ret_5d", "Michigan_Sentiment_ret_20d", "XOM_ret_1d", "AVB_AvalonBay_zscore_60d", "heston_var_ev_h7", "DAX_Germany_zscore_60d", "SLB_Schlumberger_ret_5d", "LLY_zscore_60d", "TED_Spread_vol_20d", "Nikkei_Japan_vol_20d", "PLD_Prologis_ret_5d", "GE_ret_1d", "EWM_Malaysia_ret_1d", "MSTR_Bitcoin3_ret_5d", "AMT_AmericanTower_ret_1d", "HD_zscore_60d"], "is_new": true}, {"model_id": "new_h5_NORMAL_LogisticRegression_N30_t2", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 5, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "DHR_ret_1d", "T_ret_1d", "hmm_p_stress", "Core_CPI_zscore_60d", "ORCL_zscore_60d", "INTC_ret_5d", "3M_vol_20d", "Nikkei_Japan_zscore_60d", "AVB_AvalonBay_zscore_60d", "VRP_ma5", "HUM_Humana_ret_5d", "PG_ret_20d", "BTI_BritishAmerican_ret_20d", "TGT_Target_zscore_60d", "SLB_Schlumberger_ret_5d", "SPY_zscore_60d", "AMD_ret_5d", "MS_MorganStanley_ret_1d", "LLY_zscore_60d", "Brent_Oil_FRED_ret_20d", "EWM_Malaysia_zscore_60d", "CI_Cigna_vol_20d", "US7Y_Rate_ret_20d", "DAX_Germany_vol_20d", "GILD_Gilead_ret_20d", "EMR_Emerson_ret_20d", "US3M_Rate_vol_20d", "NEE_NextEra_ret_20d"], "is_new": true}, {"model_id": "new_h5_NORMAL_LogisticRegression_N30_t3", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 5, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWY_Korea_zscore_60d", "MS_MorganStanley_ret_5d", "T_ret_1d", "MS_MorganStanley_ret_1d", "EWG_Germany_vol_20d", "TM_Telephone_ret_1d", "MSTR_Bitcoin3_ret_1d", "HUM_Humana_ret_5d", "spx_momentum_3d", "Brent_Oil_FRED_ret_5d", "BLK_BlackRock_zscore_60d", "PCAR_PaccarInc_ret_5d", "spx_vol_5d", "CPB_CampbellSoup_ret_5d", "EOG_EOGResources_ret_5d", "PAYX_Paychex_zscore_60d", "XOM_ret_20d", "WTI_Oil_FRED_zscore_60d", "EXC_Exelon_ret_1d", "DAX_Germany_vol_20d", "PPL_PPL_ret_1d", "NWL_Newell_ret_20d", "IWM_SmallCap_vol_20d", "INTC_ret_5d", "SO_SouthernCo_ret_5d", "US1Y_Rate_ret_20d", "SLB_Schlumberger_ret_5d", "DE_Deere_ret_5d"], "is_new": true}, {"model_id": "new_h5_NORMAL_LogisticRegression_N30_t4", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 5, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "SBUX_ret_5d", "ASX_Australia_ret_5d", "HUM_Humana_ret_5d", "XOM_ret_1d", "CPB_CampbellSoup_ret_5d", "HD_ret_20d", "heston_var_ev_h5", "Core_PCE_zscore_60d", "T10Y2Y_Spread_ret_5d", "LOW_Lowes_ret_5d", "NOC_Northrop_ret_20d", "DAX_Germany_zscore_60d", "EWQ_France_ret_20d", "spx_momentum_3d", "EWG_Germany_vol_20d", "TGT_Target_zscore_60d", "XLY_Disc_vol_20d", "CTAS_Cintas_vol_20d", "M_Macys_vol_20d", "SJM_JM_Smucker_ret_1d", "EQIX_Equinix_ret_5d", "US6M_Rate_ret_20d", "AXP_Amex_vol_20d", "vix_acceleration_1d", "XLF_Fin_vol_20d", "PAYX_Paychex_zscore_60d", "DHR_ret_1d", "HangSeng_HK_vol_20d"], "is_new": true}, {"model_id": "new_h5_NORMAL_LogisticRegression_N30_t5", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 5, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "Core_PCE_zscore_60d", "XLK_Tech_zscore_60d", "LOW_Lowes_ret_20d", "CLX_Clorox_vol_20d", "EWC_Canada_zscore_60d", "BA_ret_1d", "SBUX_zscore_60d", "HangSeng_HK_ret_1d", "IWM_SmallCap_vol_20d", "LUV_SouthwestAir_ret_5d", "DIS_vol_20d", "WTI_Oil_FRED_zscore_60d", "ASX_Australia_ret_5d", "AMD_ret_1d", "spx_abs_ret_max_5d", "LMT_LockheedMartin_ret_1d", "EOG_EOGResources_ret_5d", "AMT_AmericanTower_ret_1d", "heston_ev_h3", "Retail_Sales_zscore_60d", "XOM_ret_1d", "US1Y_Rate_ret_20d", "XLV_Health_zscore_60d", "US6M_Rate_ret_20d", "DE_Deere_ret_5d", "EFFR_ret_1d", "NOC_Northrop_ret_20d", "EWQ_France_zscore_60d"], "is_new": true}, {"model_id": "new_h5_NORMAL_LogisticRegression_N30_t6", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 5, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "SJM_JM_Smucker_ret_1d", "SLB_Schlumberger_ret_1d", "EWM_Malaysia_zscore_60d", "XOM_ret_20d", "vix_mean_abs_ret_5d", "BTI_BritishAmerican_ret_20d", "heston_var_ev_h5", "NEE_NextEra_ret_20d", "EXC_Exelon_zscore_60d", "DIS_vol_20d", "XLF_Fin_vol_20d", "GILD_Gilead_ret_20d", "Michigan_Sentiment_ret_20d", "CPB_CampbellSoup_ret_5d", "heston_ev_h3", "LMT_LockheedMartin_vol_20d", "SBUX_zscore_60d", "TM_Telephone_ret_1d", "DAX_Germany_vol_20d", "ENB_EnbridgeInc_ret_1d", "EWA_Australia_ret_1d", "VRP_ma5", "Core_PCE_zscore_60d", "NVDA_vol_20d", "AORD_AUS_zscore_60d", "EWC_Canada_zscore_60d", "BA_ret_1d", "TM_Telephone_vol_20d"], "is_new": true}, {"model_id": "new_h5_NORMAL_LogisticRegression_N30_t7", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 5, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWQ_France_zscore_60d", "CCI_CrownCastle_vol_20d", "CLX_Clorox_vol_20d", "HD_zscore_60d", "AMT_AmericanTower_ret_1d", "EOG_EOGResources_ret_5d", "CPB_CampbellSoup_ret_20d", "LMT_LockheedMartin_ret_1d", "EWH_HongKong_ret_5d", "AXP_Amex_vol_20d", "SLB_Schlumberger_ret_5d", "AMD_ret_5d", "HD_ret_20d", "BDX_Becton_Dickinson_ret_20d", "EMR_Emerson_ret_20d", "PFE_ret_1d", "NFCI_ret_5d", "Brent_Oil_FRED_ret_20d", "PLD_Prologis_ret_5d", "SBUX_zscore_60d", "TXN_vol_20d", "CPB_CampbellSoup_ret_5d", "hmm_p_stress", "SO_SouthernCo_ret_5d", "EWY_Korea_ret_20d", "EWS_Singapore_ret_5d", "DE_Deere_ret_5d", "EWL_Switzerland_zscore_60d"], "is_new": true}, {"model_id": "new_h5_STRESS_XGBoost_N5_t0", "algo": "XGBoost", "regime": "STRESS", "horizon": 5, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "spx_vol_5d", "DIS_vol_20d", "EWG_Germany_vol_20d"], "is_new": true}, {"model_id": "new_h5_STRESS_XGBoost_N5_t1", "algo": "XGBoost", "regime": "STRESS", "horizon": 5, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "SPY_zscore_60d", "CTAS_Cintas_vol_20d", "MS_MorganStanley_ret_1d"], "is_new": true}, {"model_id": "new_h5_STRESS_XGBoost_N5_t2", "algo": "XGBoost", "regime": "STRESS", "horizon": 5, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "QQQ_vol_20d", "NOC_Northrop_ret_20d", "LOW_Lowes_ret_5d"], "is_new": true}, {"model_id": "new_h5_STRESS_XGBoost_N5_t3", "algo": "XGBoost", "regime": "STRESS", "horizon": 5, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "AMT_AmericanTower_ret_1d", "Retail_Sales_zscore_60d", "NWL_Newell_ret_20d"], "is_new": true}, {"model_id": "new_h5_STRESS_XGBoost_N5_t4", "algo": "XGBoost", "regime": "STRESS", "horizon": 5, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "NEE_NextEra_ret_20d", "EWL_Switzerland_vol_20d", "INTC_ret_1d"], "is_new": true}, {"model_id": "new_h5_STRESS_XGBoost_N5_t5", "algo": "XGBoost", "regime": "STRESS", "horizon": 5, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "ES_Evergy_ret_1d", "CI_Cigna_vol_20d", "AMT_AmericanTower_ret_1d"], "is_new": true}, {"model_id": "new_h5_STRESS_XGBoost_N5_t6", "algo": "XGBoost", "regime": "STRESS", "horizon": 5, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "BTI_BritishAmerican_ret_5d", "AXP_Amex_ret_20d", "MS_MorganStanley_ret_1d"], "is_new": true}, {"model_id": "new_h5_STRESS_XGBoost_N5_t7", "algo": "XGBoost", "regime": "STRESS", "horizon": 5, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "3M_vol_20d", "AMZN_ret_5d", "US3Y_Rate_ret_5d"], "is_new": true}, {"model_id": "new_h5_STRESS_XGBoost_N8_t0", "algo": "XGBoost", "regime": "STRESS", "horizon": 5, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWY_Korea_zscore_60d", "LOW_Lowes_ret_20d", "DAX_Germany_vol_20d", "MSTR_Bitcoin3_ret_1d", "CPB_CampbellSoup_ret_20d", "VVIX_ret_20d"], "is_new": true}, {"model_id": "new_h5_STRESS_XGBoost_N8_t1", "algo": "XGBoost", "regime": "STRESS", "horizon": 5, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "PG_ret_20d", "3M_ret_5d", "XOM_ret_1d", "DOW_Price_zscore_60d", "EWL_Switzerland_vol_20d", "SLB_Schlumberger_ret_5d"], "is_new": true}, {"model_id": "new_h5_STRESS_XGBoost_N8_t2", "algo": "XGBoost", "regime": "STRESS", "horizon": 5, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "XOM_ret_20d", "EWM_Malaysia_ret_1d", "EOG_EOGResources_ret_5d", "EWS_Singapore_ret_5d", "MSTR_Bitcoin3_ret_20d", "ORCL_zscore_60d"], "is_new": true}, {"model_id": "new_h5_STRESS_XGBoost_N8_t3", "algo": "XGBoost", "regime": "STRESS", "horizon": 5, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "HangSeng_HK_ret_1d", "DIS_vol_20d", "LMT_LockheedMartin_vol_20d", "BTI_BritishAmerican_ret_20d", "SJM_JM_Smucker_ret_1d", "BLK_BlackRock_zscore_60d"], "is_new": true}, {"model_id": "new_h5_STRESS_XGBoost_N8_t4", "algo": "XGBoost", "regime": "STRESS", "horizon": 5, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EFFR_vol_20d", "Core_PCE_zscore_60d", "MRK_Merck_zscore_60d", "LMT_LockheedMartin_ret_1d", "DE_Deere_vol_20d", "FedFunds_zscore_60d"], "is_new": true}, {"model_id": "new_h5_STRESS_XGBoost_N8_t5", "algo": "XGBoost", "regime": "STRESS", "horizon": 5, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "DAX_Germany_vol_20d", "EWG_Germany_vol_20d", "AMD_ret_5d", "CPB_CampbellSoup_vol_20d", "Michigan_Sentiment_ret_20d", "CMCSA_ret_1d"], "is_new": true}, {"model_id": "new_h5_STRESS_XGBoost_N8_t6", "algo": "XGBoost", "regime": "STRESS", "horizon": 5, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "CPB_CampbellSoup_vol_20d", "EQR_Equity_ret_1d", "BTI_BritishAmerican_ret_20d", "spx_vol_5d", "DE_Deere_ret_5d", "MSTR_Bitcoin3_ret_5d"], "is_new": true}, {"model_id": "new_h5_STRESS_XGBoost_N8_t7", "algo": "XGBoost", "regime": "STRESS", "horizon": 5, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "Core_PCE_zscore_60d", "AXP_Amex_vol_20d", "DHR_ret_1d", "US3Y_Rate_ret_5d", "MS_MorganStanley_ret_1d", "PPL_PPL_ret_1d"], "is_new": true}, {"model_id": "new_h5_STRESS_XGBoost_N10_t0", "algo": "XGBoost", "regime": "STRESS", "horizon": 5, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "SLB_Schlumberger_ret_5d", "AMZN_ret_5d", "EWA_Australia_zscore_60d", "HD_ret_20d", "US3Y_Rate_ret_5d", "CPB_CampbellSoup_vol_20d", "ORCL_vol_20d", "EWM_Malaysia_zscore_60d"], "is_new": true}, {"model_id": "new_h5_STRESS_XGBoost_N10_t1", "algo": "XGBoost", "regime": "STRESS", "horizon": 5, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "LUV_SouthwestAir_ret_5d", "LOW_Lowes_ret_5d", "PAYX_Paychex_ret_20d", "IBEX_Spain_ret_20d", "US5Y_Rate_ret_5d", "AORD_AUS_zscore_60d", "TM_Telephone_vol_20d", "JNJ_ret_1d"], "is_new": true}, {"model_id": "new_h5_STRESS_XGBoost_N10_t2", "algo": "XGBoost", "regime": "STRESS", "horizon": 5, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "SBUX_zscore_60d", "PAYX_Paychex_ret_20d", "CPB_CampbellSoup_zscore_60d", "INTC_ret_5d", "IYR_US_REIT2_zscore_60d", "NEE_NextEra_ret_20d", "HD_zscore_60d", "T10Y2Y_Spread_ret_5d"], "is_new": true}, {"model_id": "new_h5_STRESS_XGBoost_N10_t3", "algo": "XGBoost", "regime": "STRESS", "horizon": 5, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EFFR_ret_1d", "DOW_Price_zscore_60d", "FedFunds_zscore_60d", "MS_MorganStanley_ret_5d", "VOD_Vodafone_zscore_60d", "MSTR_Bitcoin3_ret_5d", "EWM_Malaysia_zscore_60d", "CPB_CampbellSoup_zscore_60d"], "is_new": true}, {"model_id": "new_h5_STRESS_XGBoost_N10_t4", "algo": "XGBoost", "regime": "STRESS", "horizon": 5, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "3M_ret_5d", "VOD_Vodafone_zscore_60d", "XLV_Health_zscore_60d", "EWC_Canada_zscore_60d", "heston_var_ev_h7", "gjr_condvar_h1", "EFFR_vol_20d", "MS_MorganStanley_ret_1d"], "is_new": true}, {"model_id": "new_h5_STRESS_XGBoost_N10_t5", "algo": "XGBoost", "regime": "STRESS", "horizon": 5, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "WTI_Oil_FRED_zscore_60d", "DOW_Price_zscore_60d", "Nikkei_Japan_zscore_60d", "EFFR_vol_20d", "AMGN_Amgen_ret_1d", "HD_ret_20d", "SO_SouthernCo_ret_5d", "DE_Deere_vol_20d"], "is_new": true}, {"model_id": "new_h5_STRESS_XGBoost_N10_t6", "algo": "XGBoost", "regime": "STRESS", "horizon": 5, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "DIS_vol_20d", "BTI_BritishAmerican_ret_5d", "PAYX_Paychex_zscore_60d", "EWH_HongKong_ret_5d", "TM_Telephone_vol_20d", "CTAS_Cintas_vol_20d", "AXP_Amex_vol_20d", "spx_momentum_3d"], "is_new": true}, {"model_id": "new_h5_STRESS_XGBoost_N10_t7", "algo": "XGBoost", "regime": "STRESS", "horizon": 5, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "XOM_ret_20d", "EQR_Equity_ret_1d", "HD_ret_20d", "Retail_Sales_zscore_60d", "CPB_CampbellSoup_ret_5d", "JNJ_ret_1d", "spx_momentum_3d", "EMR_Emerson_ret_20d"], "is_new": true}, {"model_id": "new_h5_STRESS_XGBoost_N12_t0", "algo": "XGBoost", "regime": "STRESS", "horizon": 5, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "XLV_Health_zscore_60d", "BLK_BlackRock_zscore_60d", "EWA_Australia_ret_1d", "CPB_CampbellSoup_vol_20d", "MS_MorganStanley_ret_1d", "US3M_Rate_zscore_60d", "PAYX_Paychex_vol_20d", "EQIX_Equinix_ret_5d", "T10Y2Y_Spread_ret_5d", "TM_Telephone_vol_20d"], "is_new": true}, {"model_id": "new_h5_STRESS_XGBoost_N12_t1", "algo": "XGBoost", "regime": "STRESS", "horizon": 5, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "PFE_ret_1d", "EWQ_France_zscore_60d", "HD_zscore_60d", "EWM_Malaysia_vol_20d", "CI_Cigna_vol_20d", "US3Y_Rate_ret_5d", "AMT_AmericanTower_ret_1d", "T_ret_1d", "LMT_LockheedMartin_vol_20d", "XOM_ret_20d"], "is_new": true}, {"model_id": "new_h5_STRESS_XGBoost_N12_t2", "algo": "XGBoost", "regime": "STRESS", "horizon": 5, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "XLY_Disc_vol_20d", "SPY_zscore_60d", "ORCL_vol_20d", "PG_ret_20d", "Industrial_Production_zscore_60d", "T_ret_1d", "VVIX_ret_20d", "QQQ_vol_20d", "BLK_BlackRock_zscore_60d", "CMCSA_ret_1d"], "is_new": true}, {"model_id": "new_h5_STRESS_XGBoost_N12_t3", "algo": "XGBoost", "regime": "STRESS", "horizon": 5, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "spx_vol_5d", "XOM_ret_20d", "SCHW_Schwab_ret_5d", "AVB_AvalonBay_zscore_60d", "XLY_Disc_vol_20d", "MO_AltriaMG_ret_1d", "MRK_Merck_zscore_60d", "QQQ_vol_20d", "PCAR_PaccarInc_ret_5d", "hmm_p_stress"], "is_new": true}, {"model_id": "new_h5_STRESS_XGBoost_N12_t4", "algo": "XGBoost", "regime": "STRESS", "horizon": 5, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWM_Malaysia_zscore_60d", "NOC_Northrop_ret_20d", "T_ret_1d", "TED_Spread_vol_20d", "VVIX_ret_20d", "BA_ret_1d", "US1Y_Rate_ret_20d", "US5Y_Rate_ret_5d", "Nikkei_Japan_zscore_60d", "GE_ret_1d"], "is_new": true}, {"model_id": "new_h5_STRESS_XGBoost_N12_t5", "algo": "XGBoost", "regime": "STRESS", "horizon": 5, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "ASX_Australia_ret_5d", "CPB_CampbellSoup_vol_20d", "TM_Telephone_vol_20d", "PPL_PPL_ret_1d", "SBUX_zscore_60d", "NFCI_ret_5d", "PAYX_Paychex_zscore_60d", "CCI_CrownCastle_vol_20d", "EMR_Emerson_ret_20d", "US3Y_Rate_ret_5d"], "is_new": true}, {"model_id": "new_h5_STRESS_XGBoost_N12_t6", "algo": "XGBoost", "regime": "STRESS", "horizon": 5, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "MO_AltriaMG_ret_1d", "EWL_Switzerland_zscore_60d", "gjr_condvar_h1", "3M_ret_5d", "XLV_Health_zscore_60d", "CPB_CampbellSoup_zscore_60d", "DOW_Price_zscore_60d", "SCHW_Schwab_ret_5d", "BDX_Becton_Dickinson_ret_20d", "PAYX_Paychex_ret_20d"], "is_new": true}, {"model_id": "new_h5_STRESS_XGBoost_N12_t7", "algo": "XGBoost", "regime": "STRESS", "horizon": 5, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "BA_ret_1d", "NEE_NextEra_ret_20d", "heston_var_ev_h7", "DOW_Price_zscore_60d", "SJM_JM_Smucker_ret_1d", "DAX_Germany_zscore_60d", "EXC_Exelon_zscore_60d", "HD_ret_1d", "PCAR_PaccarInc_ret_5d", "AMD_ret_5d"], "is_new": true}, {"model_id": "new_h5_STRESS_XGBoost_N15_t0", "algo": "XGBoost", "regime": "STRESS", "horizon": 5, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "LUV_SouthwestAir_ret_5d", "AMD_ret_5d", "PCAR_PaccarInc_ret_5d", "BTI_BritishAmerican_ret_20d", "Nikkei_Japan_zscore_60d", "DIS_vol_20d", "EWM_Malaysia_zscore_60d", "VRP_ma5", "EFFR_ret_1d", "LOW_Lowes_ret_20d", "DE_Deere_vol_20d", "ORCL_vol_20d", "LLY_zscore_60d"], "is_new": true}, {"model_id": "new_h5_STRESS_XGBoost_N15_t1", "algo": "XGBoost", "regime": "STRESS", "horizon": 5, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "Industrial_Production_zscore_60d", "PAYX_Paychex_ret_20d", "NEE_NextEra_ret_20d", "vix_acceleration_1d", "SO_SouthernCo_ret_5d", "DAX_Germany_vol_20d", "CTAS_Cintas_vol_20d", "EMR_Emerson_ret_20d", "3M_vol_20d", "ASX_Australia_vol_20d", "LOW_Lowes_ret_20d", "NVDA_vol_20d", "ITT_ITTInc_ret_5d"], "is_new": true}, {"model_id": "new_h5_STRESS_XGBoost_N15_t2", "algo": "XGBoost", "regime": "STRESS", "horizon": 5, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "AXP_Amex_vol_20d", "vix_acceleration_1d", "EWY_Korea_zscore_60d", "AMT_AmericanTower_ret_1d", "SBUX_ret_5d", "HD_ret_1d", "EWC_Canada_zscore_60d", "XOM_ret_20d", "VVIX_ret_20d", "EWG_Germany_vol_20d", "TED_Spread_zscore_60d", "PFE_ret_1d", "EFFR_vol_20d"], "is_new": true}, {"model_id": "new_h5_STRESS_XGBoost_N15_t3", "algo": "XGBoost", "regime": "STRESS", "horizon": 5, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "spx_abs_ret_max_5d", "LOW_Lowes_ret_5d", "XLF_Fin_vol_20d", "EWJ_Japan_vol_20d", "EWM_Malaysia_vol_20d", "LMT_LockheedMartin_vol_20d", "XLY_Disc_vol_20d", "EQIX_Equinix_ret_5d", "HD_ret_5d", "ENB_EnbridgeInc_ret_1d", "EWA_Australia_zscore_60d", "US1Y_Rate_ret_5d", "HangSeng_HK_ret_1d"], "is_new": true}, {"model_id": "new_h5_STRESS_XGBoost_N15_t4", "algo": "XGBoost", "regime": "STRESS", "horizon": 5, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "TXN_vol_20d", "VOD_Vodafone_zscore_60d", "AMD_ret_5d", "CPB_CampbellSoup_ret_20d", "PPL_PPL_ret_1d", "BTI_BritishAmerican_ret_20d", "ASX_Australia_vol_20d", "MS_MorganStanley_zscore_60d", "EWG_Germany_ret_20d", "SBUX_vol_20d", "TED_Spread_vol_20d", "Brent_Oil_FRED_ret_5d", "DAX_Germany_vol_20d"], "is_new": true}, {"model_id": "new_h5_STRESS_XGBoost_N15_t5", "algo": "XGBoost", "regime": "STRESS", "horizon": 5, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "US7Y_Rate_ret_20d", "JNJ_ret_1d", "EXC_Exelon_zscore_60d", "EWA_Australia_ret_1d", "GILD_Gilead_ret_20d", "XOM_ret_20d", "IYR_US_REIT2_zscore_60d", "SJM_JM_Smucker_ret_1d", "Brent_Oil_FRED_ret_5d", "DE_Deere_vol_20d", "US30Y_Rate_ret_20d", "HD_ret_20d", "BDX_Becton_Dickinson_ret_20d"], "is_new": true}, {"model_id": "new_h5_STRESS_XGBoost_N15_t6", "algo": "XGBoost", "regime": "STRESS", "horizon": 5, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "US1Y_Rate_ret_20d", "TED_Spread_zscore_60d", "PG_ret_20d", "3M_vol_20d", "EWY_Korea_zscore_60d", "XLV_Health_zscore_60d", "CMCSA_ret_1d", "US3M_Rate_zscore_60d", "EWC_Canada_zscore_60d", "Core_CPI_zscore_60d", "T_ret_1d", "hmm_p_stress", "EWL_Switzerland_zscore_60d"], "is_new": true}, {"model_id": "new_h5_STRESS_XGBoost_N15_t7", "algo": "XGBoost", "regime": "STRESS", "horizon": 5, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "CPB_CampbellSoup_vol_20d", "US3M_Rate_zscore_60d", "IYM_BasicMaterials_ret_20d", "BA_ret_1d", "XLV_Health_zscore_60d", "NOC_Northrop_ret_20d", "DHR_ret_1d", "EWA_Australia_ret_1d", "EWY_Korea_ret_20d", "HangSeng_HK_vol_20d", "GE_ret_1d", "AMZN_ret_5d", "LUV_SouthwestAir_ret_5d"], "is_new": true}, {"model_id": "new_h5_STRESS_XGBoost_N20_t0", "algo": "XGBoost", "regime": "STRESS", "horizon": 5, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWC_Canada_zscore_60d", "EWH_HongKong_ret_5d", "PAYX_Paychex_vol_20d", "T10Y2Y_Spread_ret_5d", "SBUX_vol_20d", "EWG_Germany_vol_20d", "vix_mean_abs_ret_5d", "spx_momentum_3d", "EWM_Malaysia_vol_20d", "IWM_SmallCap_vol_20d", "TED_Spread_vol_20d", "BLK_BlackRock_zscore_60d", "T_ret_1d", "EMR_Emerson_ret_20d", "XLK_Tech_zscore_60d", "gjr_condvar_h1", "US7Y_Rate_ret_20d", "TED_Spread_zscore_60d"], "is_new": true}, {"model_id": "new_h5_STRESS_XGBoost_N20_t1", "algo": "XGBoost", "regime": "STRESS", "horizon": 5, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "SPY_zscore_60d", "TM_Telephone_ret_1d", "AMD_ret_5d", "ITT_ITTInc_ret_5d", "XLV_Health_zscore_60d", "IYR_US_REIT2_zscore_60d", "BTI_BritishAmerican_ret_5d", "LMT_LockheedMartin_ret_1d", "JNJ_ret_1d", "NFCI_ret_5d", "SBUX_vol_20d", "GILD_Gilead_ret_20d", "Michigan_Sentiment_ret_20d", "HangSeng_HK_ret_5d", "DE_Deere_vol_20d", "EQR_Equity_ret_1d", "PG_ret_20d", "EWC_Canada_zscore_60d"], "is_new": true}, {"model_id": "new_h5_STRESS_XGBoost_N20_t2", "algo": "XGBoost", "regime": "STRESS", "horizon": 5, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "MS_MorganStanley_ret_5d", "HUM_Humana_ret_5d", "PG_ret_20d", "ASX_Australia_vol_20d", "HD_ret_1d", "TED_Spread_vol_20d", "Retail_Sales_zscore_60d", "BLK_BlackRock_zscore_60d", "DHR_ret_1d", "TED_Spread_zscore_60d", "XOM_ret_20d", "AXP_Amex_ret_20d", "GE_ret_1d", "vix_mean_abs_ret_5d", "TM_Telephone_vol_20d", "XLK_Tech_zscore_60d", "CPB_CampbellSoup_zscore_60d", "XLB_Materials_zscore_60d"], "is_new": true}, {"model_id": "new_h5_STRESS_XGBoost_N20_t3", "algo": "XGBoost", "regime": "STRESS", "horizon": 5, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "NWL_Newell_ret_20d", "EWJ_Japan_vol_20d", "EWC_Canada_zscore_60d", "heston_ev_h3", "spx_momentum_3d", "PCAR_PaccarInc_ret_5d", "SBUX_zscore_60d", "XLB_Materials_zscore_60d", "US3Y_Rate_ret_5d", "EWQ_France_zscore_60d", "SLB_Schlumberger_ret_5d", "AMGN_Amgen_ret_1d", "EXC_Exelon_ret_1d", "PAYX_Paychex_vol_20d", "AVB_AvalonBay_zscore_60d", "Core_CPI_zscore_60d", "EWS_Singapore_ret_5d", "AMD_ret_1d"], "is_new": true}, {"model_id": "new_h5_STRESS_XGBoost_N20_t4", "algo": "XGBoost", "regime": "STRESS", "horizon": 5, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "GE_ret_1d", "VRP_ma5", "CPB_CampbellSoup_zscore_60d", "VOD_Vodafone_zscore_60d", "CPB_CampbellSoup_ret_5d", "SCHW_Schwab_ret_5d", "VVIX_ret_20d", "US3M_Rate_vol_20d", "PLD_Prologis_ret_5d", "AMT_AmericanTower_ret_1d", "IYM_BasicMaterials_ret_20d", "PCAR_PaccarInc_ret_5d", "US6M_Rate_ret_20d", "PAYX_Paychex_vol_20d", "EWL_Switzerland_vol_20d", "SLB_Schlumberger_ret_1d", "BA_ret_1d", "EOG_EOGResources_vol_20d"], "is_new": true}, {"model_id": "new_h5_STRESS_XGBoost_N20_t5", "algo": "XGBoost", "regime": "STRESS", "horizon": 5, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "ENB_EnbridgeInc_ret_1d", "EQR_Equity_ret_1d", "INTC_ret_5d", "MO_AltriaMG_ret_1d", "SBUX_vol_20d", "T10Y2Y_Spread_ret_5d", "HD_ret_20d", "Brent_Oil_FRED_ret_20d", "US3M_Rate_vol_20d", "AMD_ret_1d", "EWY_Korea_ret_20d", "DOW_Price_zscore_60d", "DAX_Germany_zscore_60d", "DE_Deere_vol_20d", "Core_CPI_zscore_60d", "EWL_Switzerland_zscore_60d", "SLB_Schlumberger_ret_1d", "SLB_Schlumberger_ret_5d"], "is_new": true}, {"model_id": "new_h5_STRESS_XGBoost_N20_t6", "algo": "XGBoost", "regime": "STRESS", "horizon": 5, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "AORD_AUS_zscore_60d", "PPL_PPL_ret_1d", "AMD_ret_5d", "XLV_Health_zscore_60d", "HangSeng_HK_vol_20d", "CCI_CrownCastle_vol_20d", "US6M_Rate_ret_20d", "ITT_ITTInc_ret_5d", "SCHW_Schwab_ret_5d", "EFFR_vol_20d", "EWY_Korea_zscore_60d", "CLX_Clorox_vol_20d", "NFCI_ret_5d", "US1Y_Rate_ret_5d", "JNJ_ret_1d", "SLB_Schlumberger_ret_1d", "spx_vol_5d", "heston_var_ev_h3"], "is_new": true}, {"model_id": "new_h5_STRESS_XGBoost_N20_t7", "algo": "XGBoost", "regime": "STRESS", "horizon": 5, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "US3M_Rate_zscore_60d", "SLB_Schlumberger_ret_5d", "spx_vol_5d", "Brent_Oil_FRED_ret_20d", "XOM_ret_20d", "ASX_Australia_vol_20d", "LMT_LockheedMartin_ret_1d", "EWG_Germany_ret_20d", "IBEX_Spain_ret_20d", "AMD_ret_1d", "LUV_SouthwestAir_ret_5d", "EWL_Switzerland_zscore_60d", "HUM_Humana_ret_5d", "SJM_JM_Smucker_ret_1d", "HD_ret_5d", "EQR_Equity_ret_1d", "heston_var_ev_h3", "EWJ_Japan_vol_20d"], "is_new": true}, {"model_id": "new_h5_STRESS_XGBoost_N25_t0", "algo": "XGBoost", "regime": "STRESS", "horizon": 5, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "XOM_ret_1d", "US7Y_Rate_ret_20d", "EWM_Malaysia_zscore_60d", "GE_ret_1d", "LMT_LockheedMartin_vol_20d", "Michigan_Sentiment_ret_20d", "LOW_Lowes_ret_20d", "Brent_Oil_FRED_ret_20d", "EWM_Malaysia_ret_1d", "VOD_Vodafone_zscore_60d", "AMZN_ret_5d", "PPL_PPL_ret_1d", "EOG_EOGResources_ret_5d", "MS_MorganStanley_ret_1d", "PCAR_PaccarInc_ret_5d", "HangSeng_HK_ret_1d", "ITT_ITTInc_ret_5d", "EWY_Korea_ret_20d", "heston_var_ev_h3", "MSTR_Bitcoin3_ret_1d", "DE_Deere_ret_5d", "INTC_ret_5d", "US3Y_Rate_ret_5d"], "is_new": true}, {"model_id": "new_h5_STRESS_XGBoost_N25_t1", "algo": "XGBoost", "regime": "STRESS", "horizon": 5, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "spx_abs_ret_max_5d", "EQR_Equity_ret_1d", "SBUX_ret_5d", "CPB_CampbellSoup_ret_5d", "INTC_ret_5d", "ES_Evergy_ret_1d", "DHR_ret_1d", "PAYX_Paychex_zscore_60d", "EWQ_France_ret_20d", "spx_vol_5d", "EMR_Emerson_ret_20d", "heston_var_ev_h7", "NFCI_ret_5d", "EWG_Germany_ret_20d", "TED_Spread_zscore_60d", "NWL_Newell_ret_20d", "CTAS_Cintas_vol_20d", "PLD_Prologis_ret_5d", "XLF_Fin_vol_20d", "MSTR_Bitcoin3_ret_20d", "US30Y_Rate_ret_20d", "BTI_BritishAmerican_ret_20d", "EWA_Australia_zscore_60d"], "is_new": true}, {"model_id": "new_h5_STRESS_XGBoost_N25_t2", "algo": "XGBoost", "regime": "STRESS", "horizon": 5, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "SCHW_Schwab_ret_5d", "SLB_Schlumberger_ret_1d", "CPB_CampbellSoup_ret_20d", "AORD_AUS_zscore_60d", "LOW_Lowes_ret_20d", "EFFR_ret_1d", "HangSeng_HK_ret_1d", "heston_var_ev_h5", "ITT_ITTInc_ret_5d", "AMD_ret_5d", "EWA_Australia_ret_1d", "INTC_ret_1d", "PCAR_PaccarInc_ret_5d", "TM_Telephone_ret_1d", "EWJ_Japan_vol_20d", "LUV_SouthwestAir_ret_5d", "IBEX_Spain_ret_20d", "EQIX_Equinix_ret_5d", "Brent_Oil_FRED_ret_20d", "Core_CPI_zscore_60d", "EWG_Germany_ret_20d", "Industrial_Production_zscore_60d", "QQQ_vol_20d"], "is_new": true}, {"model_id": "new_h5_STRESS_XGBoost_N25_t3", "algo": "XGBoost", "regime": "STRESS", "horizon": 5, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "FedFunds_zscore_60d", "HUM_Humana_ret_5d", "AMT_AmericanTower_ret_1d", "BLK_BlackRock_zscore_60d", "IYM_BasicMaterials_ret_20d", "PFE_ret_1d", "US7Y_Rate_ret_20d", "US3M_Rate_zscore_60d", "SBUX_vol_20d", "XLB_Materials_zscore_60d", "T_ret_1d", "HangSeng_HK_ret_5d", "EWL_Switzerland_zscore_60d", "CI_Cigna_vol_20d", "JNJ_ret_1d", "EOG_EOGResources_ret_5d", "EMR_Emerson_ret_20d", "heston_var_ev_h3", "XLY_Disc_vol_20d", "CTAS_Cintas_vol_20d", "AMD_ret_5d", "PLD_Prologis_ret_5d", "HangSeng_HK_vol_20d"], "is_new": true}, {"model_id": "new_h5_STRESS_XGBoost_N25_t4", "algo": "XGBoost", "regime": "STRESS", "horizon": 5, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "Michigan_Sentiment_ret_20d", "MO_AltriaMG_ret_1d", "EWQ_France_ret_20d", "AMD_ret_1d", "EQIX_Equinix_ret_5d", "3M_vol_20d", "AMZN_ret_5d", "NWL_Newell_ret_20d", "TM_Telephone_ret_1d", "vix_mean_abs_ret_5d", "LLY_zscore_60d", "EMR_Emerson_ret_20d", "EWA_Australia_ret_1d", "EFFR_vol_20d", "AXP_Amex_vol_20d", "HangSeng_HK_ret_5d", "DE_Deere_vol_20d", "AMT_AmericanTower_ret_1d", "HD_ret_5d", "XLY_Disc_vol_20d", "DIS_vol_20d", "HD_ret_1d", "PLD_Prologis_ret_5d"], "is_new": true}, {"model_id": "new_h5_STRESS_XGBoost_N25_t5", "algo": "XGBoost", "regime": "STRESS", "horizon": 5, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "SJM_JM_Smucker_ret_5d", "INTC_ret_1d", "AXP_Amex_vol_20d", "Brent_Oil_FRED_ret_5d", "EWG_Germany_ret_20d", "Michigan_Sentiment_ret_20d", "SBUX_ret_5d", "DHR_ret_1d", "HUM_Humana_ret_5d", "Industrial_Production_zscore_60d", "AMT_AmericanTower_ret_1d", "US30Y_Rate_ret_20d", "XLV_Health_zscore_60d", "EWQ_France_ret_20d", "DHR_vol_20d", "BA_ret_1d", "gjr_condvar_h1", "TED_Spread_zscore_60d", "INTC_ret_5d", "T_ret_1d", "PAYX_Paychex_ret_20d", "ITT_ITTInc_ret_5d", "DIS_vol_20d"], "is_new": true}, {"model_id": "new_h5_STRESS_XGBoost_N25_t6", "algo": "XGBoost", "regime": "STRESS", "horizon": 5, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "XLF_Fin_vol_20d", "HangSeng_HK_ret_1d", "GILD_Gilead_ret_20d", "MSTR_Bitcoin3_ret_1d", "HangSeng_HK_vol_20d", "EWL_Switzerland_zscore_60d", "CPB_CampbellSoup_ret_5d", "AMT_AmericanTower_ret_1d", "DAX_Germany_zscore_60d", "VOD_Vodafone_zscore_60d", "Core_CPI_zscore_60d", "DE_Deere_vol_20d", "AMZN_ret_5d", "US3M_Rate_vol_20d", "SBUX_zscore_60d", "ORCL_vol_20d", "JNJ_ret_1d", "T10Y2Y_Spread_ret_5d", "EWJ_Japan_vol_20d", "LOW_Lowes_ret_20d", "SCHW_Schwab_ret_5d", "gjr_condvar_h1", "PPL_PPL_ret_1d"], "is_new": true}, {"model_id": "new_h5_STRESS_XGBoost_N25_t7", "algo": "XGBoost", "regime": "STRESS", "horizon": 5, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "T10Y2Y_Spread_ret_5d", "VVIX_ret_20d", "EOG_EOGResources_vol_20d", "PFE_ret_1d", "US6M_Rate_ret_20d", "US5Y_Rate_ret_5d", "SCHW_Schwab_ret_5d", "XLV_Health_zscore_60d", "EWA_Australia_zscore_60d", "EWY_Korea_zscore_60d", "US1Y_Rate_ret_5d", "EWJ_Japan_vol_20d", "spx_vol_5d", "US3M_Rate_zscore_60d", "heston_ev_h3", "vix_mean_abs_ret_5d", "AMGN_Amgen_ret_1d", "heston_var_ev_h7", "FedFunds_zscore_60d", "DHR_vol_20d", "M_Macys_vol_20d", "SJM_JM_Smucker_ret_5d", "AMD_ret_1d"], "is_new": true}, {"model_id": "new_h5_STRESS_XGBoost_N30_t0", "algo": "XGBoost", "regime": "STRESS", "horizon": 5, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "CPB_CampbellSoup_ret_20d", "PLD_Prologis_ret_5d", "Retail_Sales_zscore_60d", "LMT_LockheedMartin_ret_1d", "HD_ret_1d", "PAYX_Paychex_zscore_60d", "spx_momentum_3d", "ASX_Australia_ret_5d", "EWL_Switzerland_zscore_60d", "Brent_Oil_FRED_ret_5d", "NVDA_vol_20d", "SO_SouthernCo_ret_5d", "DAX_Germany_vol_20d", "LOW_Lowes_ret_20d", "PFE_ret_1d", "EWA_Australia_ret_1d", "Nikkei_Japan_vol_20d", "XLF_Fin_vol_20d", "IBEX_Spain_ret_20d", "AMD_ret_5d", "NOC_Northrop_ret_20d", "US30Y_Rate_ret_20d", "3M_vol_20d", "Core_CPI_zscore_60d", "EOG_EOGResources_vol_20d", "AMZN_ret_5d", "AMGN_Amgen_ret_1d", "EWH_HongKong_ret_5d"], "is_new": true}, {"model_id": "new_h5_STRESS_XGBoost_N30_t1", "algo": "XGBoost", "regime": "STRESS", "horizon": 5, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "BDX_Becton_Dickinson_ret_20d", "HangSeng_HK_vol_20d", "PAYX_Paychex_ret_20d", "Michigan_Sentiment_ret_20d", "Industrial_Production_zscore_60d", "XOM_ret_1d", "IYR_US_REIT2_zscore_60d", "HUM_Humana_ret_5d", "HD_ret_5d", "ASX_Australia_vol_20d", "US30Y_Rate_ret_20d", "TM_Telephone_vol_20d", "US1Y_Rate_ret_5d", "SJM_JM_Smucker_ret_5d", "heston_var_ev_h7", "AMD_ret_1d", "EWC_Canada_zscore_60d", "SBUX_zscore_60d", "EMR_Emerson_ret_20d", "BTI_BritishAmerican_ret_5d", "US3Y_Rate_ret_5d", "EWG_Germany_vol_20d", "AMZN_ret_5d", "vix_acceleration_1d", "CMCSA_ret_1d", "hmm_p_stress", "AMGN_Amgen_ret_1d", "T10Y2Y_Spread_ret_5d"], "is_new": true}, {"model_id": "new_h5_STRESS_XGBoost_N30_t2", "algo": "XGBoost", "regime": "STRESS", "horizon": 5, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "AMGN_Amgen_ret_1d", "HD_ret_20d", "NVDA_vol_20d", "EFFR_vol_20d", "BDX_Becton_Dickinson_ret_20d", "T_ret_1d", "SJM_JM_Smucker_ret_1d", "EWM_Malaysia_zscore_60d", "hmm_p_stress", "HangSeng_HK_vol_20d", "EWL_Switzerland_zscore_60d", "DHR_ret_1d", "ENB_EnbridgeInc_ret_1d", "CPB_CampbellSoup_ret_5d", "LUV_SouthwestAir_ret_5d", "MS_MorganStanley_ret_5d", "heston_var_ev_h5", "HUM_Humana_ret_5d", "EWC_Canada_zscore_60d", "SBUX_vol_20d", "spx_vol_5d", "SJM_JM_Smucker_ret_5d", "DHR_vol_20d", "LMT_LockheedMartin_ret_1d", "DAX_Germany_zscore_60d", "EWS_Singapore_ret_5d", "SLB_Schlumberger_ret_1d", "CPB_CampbellSoup_ret_20d"], "is_new": true}, {"model_id": "new_h5_STRESS_XGBoost_N30_t3", "algo": "XGBoost", "regime": "STRESS", "horizon": 5, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWH_HongKong_ret_5d", "EOG_EOGResources_ret_5d", "heston_var_ev_h7", "XLF_Fin_vol_20d", "WTI_Oil_FRED_zscore_60d", "LOW_Lowes_ret_20d", "IWM_SmallCap_vol_20d", "PAYX_Paychex_vol_20d", "LMT_LockheedMartin_vol_20d", "INTC_ret_1d", "EWQ_France_ret_20d", "TED_Spread_zscore_60d", "EWM_Malaysia_vol_20d", "LOW_Lowes_ret_5d", "AORD_AUS_zscore_60d", "ORCL_vol_20d", "LLY_zscore_60d", "EWY_Korea_zscore_60d", "Brent_Oil_FRED_ret_20d", "EWA_Australia_ret_1d", "US5Y_Rate_ret_5d", "XLV_Health_zscore_60d", "PPL_PPL_ret_1d", "GILD_Gilead_ret_20d", "XLB_Materials_zscore_60d", "EWY_Korea_ret_20d", "DAX_Germany_zscore_60d", "SBUX_vol_20d"], "is_new": true}, {"model_id": "new_h5_STRESS_XGBoost_N30_t4", "algo": "XGBoost", "regime": "STRESS", "horizon": 5, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "DHR_ret_1d", "ES_Evergy_ret_1d", "DAX_Germany_vol_20d", "DHR_vol_20d", "EWQ_France_ret_20d", "PAYX_Paychex_zscore_60d", "AMD_ret_1d", "EWM_Malaysia_vol_20d", "EFFR_ret_1d", "heston_ev_h3", "vix_mean_abs_ret_5d", "TED_Spread_vol_20d", "SJM_JM_Smucker_ret_5d", "TGT_Target_zscore_60d", "XLF_Fin_vol_20d", "QQQ_vol_20d", "BLK_BlackRock_zscore_60d", "EQR_Equity_ret_1d", "ORCL_vol_20d", "AORD_AUS_zscore_60d", "BDX_Becton_Dickinson_ret_20d", "LMT_LockheedMartin_vol_20d", "gjr_condvar_h1", "XLV_Health_zscore_60d", "NWL_Newell_ret_20d", "Core_CPI_zscore_60d", "VVIX_ret_20d", "XOM_ret_20d"], "is_new": true}, {"model_id": "new_h5_STRESS_XGBoost_N30_t5", "algo": "XGBoost", "regime": "STRESS", "horizon": 5, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "US7Y_Rate_ret_20d", "spx_momentum_3d", "SCHW_Schwab_ret_5d", "QQQ_vol_20d", "heston_var_ev_h5", "PG_ret_20d", "Industrial_Production_zscore_60d", "EWA_Australia_ret_1d", "DHR_ret_1d", "EWS_Singapore_ret_5d", "T_ret_1d", "MS_MorganStanley_ret_5d", "EWA_Australia_zscore_60d", "SBUX_ret_5d", "PPL_PPL_ret_1d", "EWG_Germany_vol_20d", "Michigan_Sentiment_ret_20d", "HUM_Humana_ret_5d", "DIS_vol_20d", "Core_CPI_zscore_60d", "PAYX_Paychex_ret_20d", "TXN_vol_20d", "IYR_US_REIT2_zscore_60d", "XLV_Health_zscore_60d", "EMR_Emerson_ret_20d", "EOG_EOGResources_ret_5d", "SO_SouthernCo_ret_5d", "AMGN_Amgen_ret_1d"], "is_new": true}, {"model_id": "new_h5_STRESS_XGBoost_N30_t6", "algo": "XGBoost", "regime": "STRESS", "horizon": 5, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "XLB_Materials_zscore_60d", "CLX_Clorox_vol_20d", "ORCL_zscore_60d", "HUM_Humana_ret_5d", "ES_Evergy_ret_1d", "EWJ_Japan_vol_20d", "US1Y_Rate_ret_20d", "BA_ret_1d", "TM_Telephone_ret_1d", "ORCL_vol_20d", "vix_mean_abs_ret_5d", "vix_acceleration_1d", "US6M_Rate_ret_20d", "CI_Cigna_vol_20d", "T10Y2Y_Spread_ret_5d", "heston_var_ev_h5", "US3Y_Rate_ret_5d", "heston_var_ev_h3", "CPB_CampbellSoup_vol_20d", "QQQ_vol_20d", "HangSeng_HK_ret_1d", "INTC_ret_5d", "IYM_BasicMaterials_ret_20d", "DE_Deere_ret_5d", "PAYX_Paychex_zscore_60d", "VOD_Vodafone_zscore_60d", "EWC_Canada_zscore_60d", "SCHW_Schwab_ret_5d"], "is_new": true}, {"model_id": "new_h5_STRESS_XGBoost_N30_t7", "algo": "XGBoost", "regime": "STRESS", "horizon": 5, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "ASX_Australia_ret_5d", "EOG_EOGResources_vol_20d", "EWG_Germany_ret_20d", "EWL_Switzerland_zscore_60d", "MSTR_Bitcoin3_ret_5d", "EFFR_vol_20d", "US30Y_Rate_ret_20d", "MSTR_Bitcoin3_ret_1d", "HD_ret_5d", "CCI_CrownCastle_vol_20d", "BTI_BritishAmerican_ret_5d", "MS_MorganStanley_ret_5d", "XLB_Materials_zscore_60d", "DAX_Germany_zscore_60d", "SBUX_zscore_60d", "PPL_PPL_ret_1d", "AXP_Amex_vol_20d", "EWS_Singapore_ret_5d", "US1Y_Rate_ret_20d", "VVIX_ret_20d", "EWM_Malaysia_zscore_60d", "Brent_Oil_FRED_ret_20d", "EWM_Malaysia_ret_1d", "Nikkei_Japan_vol_20d", "EXC_Exelon_zscore_60d", "AMT_AmericanTower_ret_1d", "XOM_ret_1d", "Nikkei_Japan_zscore_60d"], "is_new": true}, {"model_id": "new_h5_STRESS_LightGBM_N5_t0", "algo": "LightGBM", "regime": "STRESS", "horizon": 5, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "NWL_Newell_ret_20d", "TED_Spread_zscore_60d", "SPY_zscore_60d"], "is_new": true}, {"model_id": "new_h5_STRESS_LightGBM_N5_t1", "algo": "LightGBM", "regime": "STRESS", "horizon": 5, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "Brent_Oil_FRED_ret_20d", "WTI_Oil_FRED_zscore_60d", "AORD_AUS_zscore_60d"], "is_new": true}, {"model_id": "new_h5_STRESS_LightGBM_N5_t2", "algo": "LightGBM", "regime": "STRESS", "horizon": 5, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "LMT_LockheedMartin_vol_20d", "GE_ret_1d", "AMD_ret_1d"], "is_new": true}, {"model_id": "new_h5_STRESS_LightGBM_N5_t3", "algo": "LightGBM", "regime": "STRESS", "horizon": 5, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "SPY_zscore_60d", "SBUX_vol_20d", "AMT_AmericanTower_ret_1d"], "is_new": true}, {"model_id": "new_h5_STRESS_LightGBM_N5_t4", "algo": "LightGBM", "regime": "STRESS", "horizon": 5, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "SBUX_ret_5d", "GD_GeneralDynamics_zscore_60d", "HD_zscore_60d"], "is_new": true}, {"model_id": "new_h5_STRESS_LightGBM_N5_t5", "algo": "LightGBM", "regime": "STRESS", "horizon": 5, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWS_Singapore_ret_5d", "DHR_vol_20d", "TED_Spread_vol_20d"], "is_new": true}, {"model_id": "new_h5_STRESS_LightGBM_N5_t6", "algo": "LightGBM", "regime": "STRESS", "horizon": 5, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWY_Korea_ret_20d", "MS_MorganStanley_zscore_60d", "DIS_vol_20d"], "is_new": true}, {"model_id": "new_h5_STRESS_LightGBM_N5_t7", "algo": "LightGBM", "regime": "STRESS", "horizon": 5, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "M_Macys_vol_20d", "DOW_Price_zscore_60d", "AXP_Amex_vol_20d"], "is_new": true}, {"model_id": "new_h5_STRESS_LightGBM_N8_t0", "algo": "LightGBM", "regime": "STRESS", "horizon": 5, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "NEE_NextEra_ret_20d", "XLB_Materials_zscore_60d", "EWS_Singapore_ret_5d", "DAX_Germany_vol_20d", "TGT_Target_zscore_60d", "LUV_SouthwestAir_ret_5d"], "is_new": true}, {"model_id": "new_h5_STRESS_LightGBM_N8_t1", "algo": "LightGBM", "regime": "STRESS", "horizon": 5, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "T10Y2Y_Spread_ret_5d", "ORCL_vol_20d", "DHR_vol_20d", "MO_AltriaMG_ret_1d", "EWL_Switzerland_zscore_60d", "WTI_Oil_FRED_zscore_60d"], "is_new": true}, {"model_id": "new_h5_STRESS_LightGBM_N8_t2", "algo": "LightGBM", "regime": "STRESS", "horizon": 5, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "TM_Telephone_ret_1d", "SBUX_zscore_60d", "HD_ret_20d", "ES_Evergy_ret_1d", "DE_Deere_vol_20d", "EXC_Exelon_ret_1d"], "is_new": true}, {"model_id": "new_h5_STRESS_LightGBM_N8_t3", "algo": "LightGBM", "regime": "STRESS", "horizon": 5, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EXC_Exelon_ret_1d", "DHR_vol_20d", "MSTR_Bitcoin3_ret_5d", "US7Y_Rate_ret_20d", "EFFR_vol_20d", "NEE_NextEra_ret_20d"], "is_new": true}, {"model_id": "new_h5_STRESS_LightGBM_N8_t4", "algo": "LightGBM", "regime": "STRESS", "horizon": 5, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "US30Y_Rate_ret_20d", "EWH_HongKong_ret_5d", "3M_vol_20d", "EWY_Korea_zscore_60d", "CTAS_Cintas_vol_20d", "AMGN_Amgen_ret_1d"], "is_new": true}, {"model_id": "new_h5_STRESS_LightGBM_N8_t5", "algo": "LightGBM", "regime": "STRESS", "horizon": 5, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "AMD_ret_1d", "BTI_BritishAmerican_ret_5d", "VVIX_ret_20d", "EWM_Malaysia_ret_1d", "heston_var_ev_h3", "PCAR_PaccarInc_ret_5d"], "is_new": true}, {"model_id": "new_h5_STRESS_LightGBM_N8_t6", "algo": "LightGBM", "regime": "STRESS", "horizon": 5, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "INTC_ret_5d", "heston_var_ev_h5", "ORCL_vol_20d", "Michigan_Sentiment_ret_20d", "TM_Telephone_ret_1d", "AMZN_ret_5d"], "is_new": true}, {"model_id": "new_h5_STRESS_LightGBM_N8_t7", "algo": "LightGBM", "regime": "STRESS", "horizon": 5, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "PFE_ret_1d", "DAX_Germany_vol_20d", "AMT_AmericanTower_ret_1d", "PG_ret_20d", "CPB_CampbellSoup_vol_20d", "SO_SouthernCo_ret_5d"], "is_new": true}, {"model_id": "new_h5_STRESS_LightGBM_N10_t0", "algo": "LightGBM", "regime": "STRESS", "horizon": 5, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "ASX_Australia_vol_20d", "EOG_EOGResources_vol_20d", "Nikkei_Japan_vol_20d", "HangSeng_HK_ret_5d", "QQQ_vol_20d", "DE_Deere_vol_20d", "EWG_Germany_ret_20d", "Brent_Oil_FRED_ret_5d"], "is_new": true}, {"model_id": "new_h5_STRESS_LightGBM_N10_t1", "algo": "LightGBM", "regime": "STRESS", "horizon": 5, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "AXP_Amex_ret_20d", "DOW_Price_zscore_60d", "AMD_ret_5d", "PAYX_Paychex_vol_20d", "MS_MorganStanley_zscore_60d", "LOW_Lowes_ret_20d", "SBUX_vol_20d", "US3Y_Rate_ret_5d"], "is_new": true}, {"model_id": "new_h5_STRESS_LightGBM_N10_t2", "algo": "LightGBM", "regime": "STRESS", "horizon": 5, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "T10Y2Y_Spread_ret_5d", "CCI_CrownCastle_vol_20d", "EWG_Germany_vol_20d", "HangSeng_HK_ret_1d", "EWM_Malaysia_vol_20d", "HUM_Humana_ret_5d", "GE_ret_1d", "PG_ret_20d"], "is_new": true}, {"model_id": "new_h5_STRESS_LightGBM_N10_t3", "algo": "LightGBM", "regime": "STRESS", "horizon": 5, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "MS_MorganStanley_zscore_60d", "US3M_Rate_zscore_60d", "MSTR_Bitcoin3_ret_5d", "NEE_NextEra_ret_20d", "EWY_Korea_ret_20d", "PAYX_Paychex_vol_20d", "T_ret_1d", "Nikkei_Japan_vol_20d"], "is_new": true}, {"model_id": "new_h5_STRESS_LightGBM_N10_t4", "algo": "LightGBM", "regime": "STRESS", "horizon": 5, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "LMT_LockheedMartin_vol_20d", "XOM_ret_1d", "SCHW_Schwab_ret_5d", "SJM_JM_Smucker_ret_1d", "NEE_NextEra_ret_20d", "SBUX_ret_5d", "heston_var_ev_h5", "EWY_Korea_zscore_60d"], "is_new": true}, {"model_id": "new_h5_STRESS_LightGBM_N10_t5", "algo": "LightGBM", "regime": "STRESS", "horizon": 5, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "MS_MorganStanley_zscore_60d", "heston_var_ev_h5", "WTI_Oil_FRED_zscore_60d", "ENB_EnbridgeInc_ret_1d", "LMT_LockheedMartin_ret_1d", "CI_Cigna_vol_20d", "XLB_Materials_zscore_60d", "vix_acceleration_1d"], "is_new": true}, {"model_id": "new_h5_STRESS_LightGBM_N10_t6", "algo": "LightGBM", "regime": "STRESS", "horizon": 5, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "Nikkei_Japan_vol_20d", "JNJ_ret_1d", "EQIX_Equinix_ret_5d", "XLV_Health_zscore_60d", "DHR_vol_20d", "PCAR_PaccarInc_ret_5d", "vix_mean_abs_ret_5d", "US1Y_Rate_ret_5d"], "is_new": true}, {"model_id": "new_h5_STRESS_LightGBM_N10_t7", "algo": "LightGBM", "regime": "STRESS", "horizon": 5, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EXC_Exelon_zscore_60d", "T_ret_1d", "AMD_ret_1d", "GD_GeneralDynamics_zscore_60d", "Nikkei_Japan_zscore_60d", "SBUX_ret_5d", "PPL_PPL_ret_1d", "XLF_Fin_vol_20d"], "is_new": true}, {"model_id": "new_h5_STRESS_LightGBM_N12_t0", "algo": "LightGBM", "regime": "STRESS", "horizon": 5, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "BA_ret_1d", "DOW_Price_zscore_60d", "XOM_ret_20d", "PG_ret_20d", "US30Y_Rate_ret_20d", "MSTR_Bitcoin3_ret_5d", "spx_vol_5d", "EFFR_vol_20d", "EWG_Germany_ret_20d", "CPB_CampbellSoup_zscore_60d"], "is_new": true}, {"model_id": "new_h5_STRESS_LightGBM_N12_t1", "algo": "LightGBM", "regime": "STRESS", "horizon": 5, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "ASX_Australia_ret_5d", "EWA_Australia_zscore_60d", "XLF_Fin_vol_20d", "PAYX_Paychex_zscore_60d", "TED_Spread_vol_20d", "INTC_ret_1d", "CCI_CrownCastle_vol_20d", "HangSeng_HK_ret_1d", "Brent_Oil_FRED_ret_20d", "EQIX_Equinix_ret_5d"], "is_new": true}, {"model_id": "new_h5_STRESS_LightGBM_N12_t2", "algo": "LightGBM", "regime": "STRESS", "horizon": 5, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "MSTR_Bitcoin3_ret_5d", "SCHW_Schwab_ret_5d", "SBUX_vol_20d", "EWM_Malaysia_vol_20d", "NOC_Northrop_ret_20d", "TXN_vol_20d", "INTC_ret_5d", "EOG_EOGResources_vol_20d", "EWQ_France_ret_20d", "PCAR_PaccarInc_ret_5d"], "is_new": true}, {"model_id": "new_h5_STRESS_LightGBM_N12_t3", "algo": "LightGBM", "regime": "STRESS", "horizon": 5, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "DAX_Germany_vol_20d", "Core_CPI_zscore_60d", "IYR_US_REIT2_zscore_60d", "JNJ_ret_1d", "TGT_Target_zscore_60d", "HD_ret_20d", "US7Y_Rate_ret_20d", "vix_acceleration_1d", "Retail_Sales_zscore_60d", "EWA_Australia_ret_1d"], "is_new": true}, {"model_id": "new_h5_STRESS_LightGBM_N12_t4", "algo": "LightGBM", "regime": "STRESS", "horizon": 5, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "SPY_zscore_60d", "CTAS_Cintas_vol_20d", "Brent_Oil_FRED_ret_5d", "BLK_BlackRock_zscore_60d", "LOW_Lowes_ret_20d", "DHR_vol_20d", "CMCSA_ret_1d", "ENB_EnbridgeInc_ret_1d", "FedFunds_zscore_60d", "AMT_AmericanTower_ret_1d"], "is_new": true}, {"model_id": "new_h5_STRESS_LightGBM_N12_t5", "algo": "LightGBM", "regime": "STRESS", "horizon": 5, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "SBUX_ret_5d", "T10Y2Y_Spread_ret_5d", "MSTR_Bitcoin3_ret_1d", "CPB_CampbellSoup_ret_5d", "XLV_Health_zscore_60d", "TGT_Target_zscore_60d", "MS_MorganStanley_ret_1d", "MSTR_Bitcoin3_ret_20d", "HD_zscore_60d", "LMT_LockheedMartin_vol_20d"], "is_new": true}, {"model_id": "new_h5_STRESS_LightGBM_N12_t6", "algo": "LightGBM", "regime": "STRESS", "horizon": 5, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWM_Malaysia_vol_20d", "EWG_Germany_vol_20d", "DAX_Germany_vol_20d", "AXP_Amex_ret_20d", "ORCL_zscore_60d", "gjr_condvar_h1", "HangSeng_HK_ret_5d", "M_Macys_vol_20d", "AMZN_ret_5d", "EWA_Australia_zscore_60d"], "is_new": true}, {"model_id": "new_h5_STRESS_LightGBM_N12_t7", "algo": "LightGBM", "regime": "STRESS", "horizon": 5, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "PFE_ret_1d", "EWA_Australia_ret_1d", "heston_var_ev_h7", "MO_AltriaMG_ret_1d", "EWY_Korea_ret_20d", "EWQ_France_ret_20d", "US5Y_Rate_ret_5d", "NEE_NextEra_ret_20d", "FedFunds_zscore_60d", "CCI_CrownCastle_vol_20d"], "is_new": true}, {"model_id": "new_h5_STRESS_LightGBM_N15_t0", "algo": "LightGBM", "regime": "STRESS", "horizon": 5, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "SBUX_ret_5d", "MO_AltriaMG_ret_1d", "Retail_Sales_zscore_60d", "US3M_Rate_zscore_60d", "TM_Telephone_ret_1d", "US1Y_Rate_ret_20d", "AMD_ret_5d", "EOG_EOGResources_vol_20d", "PFE_ret_1d", "DHR_ret_1d", "BDX_Becton_Dickinson_ret_20d", "XLV_Health_zscore_60d", "XLF_Fin_vol_20d"], "is_new": true}, {"model_id": "new_h5_STRESS_LightGBM_N15_t1", "algo": "LightGBM", "regime": "STRESS", "horizon": 5, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "LUV_SouthwestAir_ret_5d", "Brent_Oil_FRED_ret_20d", "EFFR_ret_1d", "3M_ret_5d", "XLY_Disc_vol_20d", "TED_Spread_zscore_60d", "PCAR_PaccarInc_ret_5d", "LOW_Lowes_ret_20d", "JNJ_ret_1d", "IBEX_Spain_ret_20d", "M_Macys_vol_20d", "SJM_JM_Smucker_ret_1d", "BDX_Becton_Dickinson_ret_20d"], "is_new": true}, {"model_id": "new_h5_STRESS_LightGBM_N15_t2", "algo": "LightGBM", "regime": "STRESS", "horizon": 5, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "T_ret_1d", "PFE_ret_1d", "AXP_Amex_ret_20d", "CPB_CampbellSoup_ret_5d", "DE_Deere_ret_5d", "SPY_zscore_60d", "AMGN_Amgen_ret_1d", "US30Y_Rate_ret_20d", "ASX_Australia_ret_5d", "EWA_Australia_zscore_60d", "GILD_Gilead_ret_20d", "Core_PCE_zscore_60d", "MS_MorganStanley_ret_5d"], "is_new": true}, {"model_id": "new_h5_STRESS_LightGBM_N15_t3", "algo": "LightGBM", "regime": "STRESS", "horizon": 5, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "CMCSA_ret_1d", "AMD_ret_1d", "TED_Spread_zscore_60d", "EWG_Germany_ret_20d", "BA_ret_1d", "CPB_CampbellSoup_vol_20d", "MRK_Merck_zscore_60d", "EWA_Australia_zscore_60d", "JNJ_ret_1d", "heston_ev_h3", "DHR_ret_1d", "PAYX_Paychex_vol_20d", "US1Y_Rate_ret_5d"], "is_new": true}, {"model_id": "new_h5_STRESS_LightGBM_N15_t4", "algo": "LightGBM", "regime": "STRESS", "horizon": 5, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "ASX_Australia_vol_20d", "SLB_Schlumberger_ret_5d", "TGT_Target_zscore_60d", "NEE_NextEra_ret_20d", "EWA_Australia_zscore_60d", "US1Y_Rate_ret_5d", "spx_momentum_3d", "HD_ret_5d", "CI_Cigna_vol_20d", "WTI_Oil_FRED_zscore_60d", "MSTR_Bitcoin3_ret_5d", "DHR_ret_1d", "TXN_vol_20d"], "is_new": true}, {"model_id": "new_h5_STRESS_LightGBM_N15_t5", "algo": "LightGBM", "regime": "STRESS", "horizon": 5, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "ORCL_zscore_60d", "HD_ret_20d", "MS_MorganStanley_ret_1d", "XLY_Disc_vol_20d", "AORD_AUS_zscore_60d", "LOW_Lowes_ret_5d", "SCHW_Schwab_ret_5d", "US6M_Rate_ret_20d", "ENB_EnbridgeInc_ret_1d", "EWY_Korea_ret_20d", "EOG_EOGResources_ret_5d", "spx_vol_5d", "BTI_BritishAmerican_ret_20d"], "is_new": true}, {"model_id": "new_h5_STRESS_LightGBM_N15_t6", "algo": "LightGBM", "regime": "STRESS", "horizon": 5, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "NOC_Northrop_ret_20d", "TM_Telephone_vol_20d", "LLY_zscore_60d", "EWC_Canada_zscore_60d", "TED_Spread_zscore_60d", "XLF_Fin_vol_20d", "EWM_Malaysia_zscore_60d", "EWA_Australia_ret_1d", "IWM_SmallCap_vol_20d", "MSTR_Bitcoin3_ret_1d", "DAX_Germany_zscore_60d", "GD_GeneralDynamics_zscore_60d", "CTAS_Cintas_vol_20d"], "is_new": true}, {"model_id": "new_h5_STRESS_LightGBM_N15_t7", "algo": "LightGBM", "regime": "STRESS", "horizon": 5, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "US1Y_Rate_ret_5d", "M_Macys_vol_20d", "EWQ_France_ret_20d", "CPB_CampbellSoup_ret_5d", "TXN_vol_20d", "hmm_p_stress", "US7Y_Rate_ret_20d", "EQIX_Equinix_ret_5d", "PG_ret_20d", "SPY_zscore_60d", "MSTR_Bitcoin3_ret_20d", "XLY_Disc_vol_20d", "US30Y_Rate_ret_20d"], "is_new": true}, {"model_id": "new_h5_STRESS_LightGBM_N20_t0", "algo": "LightGBM", "regime": "STRESS", "horizon": 5, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "US3M_Rate_zscore_60d", "MSTR_Bitcoin3_ret_5d", "PCAR_PaccarInc_ret_5d", "EWG_Germany_ret_20d", "PLD_Prologis_ret_5d", "VVIX_ret_20d", "XLK_Tech_zscore_60d", "LOW_Lowes_ret_5d", "TGT_Target_zscore_60d", "T10Y2Y_Spread_ret_5d", "ASX_Australia_ret_5d", "SPY_zscore_60d", "ITT_ITTInc_ret_5d", "WTI_Oil_FRED_zscore_60d", "spx_momentum_3d", "NEE_NextEra_ret_20d", "ORCL_vol_20d", "LMT_LockheedMartin_ret_1d"], "is_new": true}, {"model_id": "new_h5_STRESS_LightGBM_N20_t1", "algo": "LightGBM", "regime": "STRESS", "horizon": 5, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "heston_var_ev_h3", "gjr_condvar_h1", "HangSeng_HK_ret_5d", "TED_Spread_vol_20d", "QQQ_vol_20d", "US1Y_Rate_ret_20d", "MSTR_Bitcoin3_ret_1d", "GD_GeneralDynamics_zscore_60d", "T10Y2Y_Spread_ret_5d", "AMZN_ret_5d", "DAX_Germany_vol_20d", "BA_ret_1d", "IYM_BasicMaterials_ret_20d", "Core_CPI_zscore_60d", "hmm_p_stress", "CMCSA_ret_1d", "SLB_Schlumberger_ret_1d", "HD_ret_20d"], "is_new": true}, {"model_id": "new_h5_STRESS_LightGBM_N20_t2", "algo": "LightGBM", "regime": "STRESS", "horizon": 5, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "3M_vol_20d", "EFFR_vol_20d", "DIS_vol_20d", "AXP_Amex_vol_20d", "LUV_SouthwestAir_ret_5d", "TGT_Target_zscore_60d", "EWQ_France_zscore_60d", "ITT_ITTInc_ret_5d", "US6M_Rate_ret_20d", "heston_ev_h3", "QQQ_vol_20d", "US1Y_Rate_ret_20d", "T10Y2Y_Spread_ret_5d", "AMT_AmericanTower_ret_1d", "EWM_Malaysia_ret_1d", "LLY_zscore_60d", "DE_Deere_vol_20d", "MSTR_Bitcoin3_ret_1d"], "is_new": true}, {"model_id": "new_h5_STRESS_LightGBM_N20_t3", "algo": "LightGBM", "regime": "STRESS", "horizon": 5, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "PAYX_Paychex_ret_20d", "MS_MorganStanley_zscore_60d", "AMGN_Amgen_ret_1d", "ORCL_vol_20d", "PCAR_PaccarInc_ret_5d", "ASX_Australia_vol_20d", "DIS_vol_20d", "LUV_SouthwestAir_ret_5d", "3M_vol_20d", "XLY_Disc_vol_20d", "CCI_CrownCastle_vol_20d", "vix_mean_abs_ret_5d", "PPL_PPL_ret_1d", "spx_abs_ret_max_5d", "XLV_Health_zscore_60d", "ENB_EnbridgeInc_ret_1d", "Michigan_Sentiment_ret_20d", "CPB_CampbellSoup_ret_5d"], "is_new": true}, {"model_id": "new_h5_STRESS_LightGBM_N20_t4", "algo": "LightGBM", "regime": "STRESS", "horizon": 5, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "BLK_BlackRock_zscore_60d", "MSTR_Bitcoin3_ret_1d", "heston_var_ev_h7", "US3Y_Rate_ret_5d", "XLK_Tech_zscore_60d", "EWQ_France_ret_20d", "EQR_Equity_ret_1d", "EWS_Singapore_ret_5d", "DHR_ret_1d", "EWA_Australia_zscore_60d", "Nikkei_Japan_zscore_60d", "SBUX_zscore_60d", "EWQ_France_zscore_60d", "M_Macys_vol_20d", "JNJ_ret_1d", "EXC_Exelon_ret_1d", "AXP_Amex_vol_20d", "CCI_CrownCastle_vol_20d"], "is_new": true}, {"model_id": "new_h5_STRESS_LightGBM_N20_t5", "algo": "LightGBM", "regime": "STRESS", "horizon": 5, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "BTI_BritishAmerican_ret_5d", "CLX_Clorox_vol_20d", "EOG_EOGResources_ret_5d", "AMT_AmericanTower_ret_1d", "DHR_vol_20d", "PG_ret_20d", "EQIX_Equinix_ret_5d", "Core_CPI_zscore_60d", "MS_MorganStanley_ret_5d", "EWQ_France_zscore_60d", "SBUX_zscore_60d", "HUM_Humana_ret_5d", "vix_mean_abs_ret_5d", "XOM_ret_20d", "CPB_CampbellSoup_ret_20d", "XLY_Disc_vol_20d", "VVIX_ret_20d", "EWM_Malaysia_ret_1d"], "is_new": true}, {"model_id": "new_h5_STRESS_LightGBM_N20_t6", "algo": "LightGBM", "regime": "STRESS", "horizon": 5, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWM_Malaysia_ret_1d", "HD_ret_1d", "AMGN_Amgen_ret_1d", "XLF_Fin_vol_20d", "IYR_US_REIT2_zscore_60d", "TM_Telephone_ret_1d", "MS_MorganStanley_zscore_60d", "EWS_Singapore_ret_5d", "INTC_ret_5d", "HD_zscore_60d", "ORCL_vol_20d", "US7Y_Rate_ret_20d", "ES_Evergy_ret_1d", "EWY_Korea_ret_20d", "DHR_ret_1d", "MSTR_Bitcoin3_ret_5d", "Nikkei_Japan_vol_20d", "EXC_Exelon_ret_1d"], "is_new": true}, {"model_id": "new_h5_STRESS_LightGBM_N20_t7", "algo": "LightGBM", "regime": "STRESS", "horizon": 5, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "VRP_ma5", "AMGN_Amgen_ret_1d", "PAYX_Paychex_vol_20d", "GILD_Gilead_ret_20d", "DE_Deere_ret_5d", "spx_vol_5d", "MSTR_Bitcoin3_ret_5d", "NWL_Newell_ret_20d", "EWG_Germany_vol_20d", "EWH_HongKong_ret_5d", "SJM_JM_Smucker_ret_5d", "XOM_ret_20d", "XOM_ret_1d", "CI_Cigna_vol_20d", "EOG_EOGResources_vol_20d", "DIS_vol_20d", "TED_Spread_zscore_60d", "AMZN_ret_5d"], "is_new": true}, {"model_id": "new_h5_STRESS_LightGBM_N25_t0", "algo": "LightGBM", "regime": "STRESS", "horizon": 5, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "BDX_Becton_Dickinson_ret_20d", "gjr_condvar_h1", "NEE_NextEra_ret_20d", "AMT_AmericanTower_ret_1d", "XOM_ret_1d", "EWC_Canada_zscore_60d", "Brent_Oil_FRED_ret_20d", "HD_ret_5d", "BTI_BritishAmerican_ret_20d", "PFE_ret_1d", "EWS_Singapore_ret_5d", "EWY_Korea_ret_20d", "MS_MorganStanley_ret_1d", "CTAS_Cintas_vol_20d", "EFFR_vol_20d", "EWM_Malaysia_zscore_60d", "VOD_Vodafone_zscore_60d", "EWL_Switzerland_vol_20d", "EWG_Germany_ret_20d", "T_ret_1d", "CMCSA_ret_1d", "VVIX_ret_20d", "EXC_Exelon_ret_1d"], "is_new": true}, {"model_id": "new_h5_STRESS_LightGBM_N25_t1", "algo": "LightGBM", "regime": "STRESS", "horizon": 5, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "CCI_CrownCastle_vol_20d", "Industrial_Production_zscore_60d", "EFFR_vol_20d", "EWQ_France_zscore_60d", "TED_Spread_zscore_60d", "JNJ_ret_1d", "US30Y_Rate_ret_20d", "US5Y_Rate_ret_5d", "ORCL_zscore_60d", "DOW_Price_zscore_60d", "LMT_LockheedMartin_ret_1d", "AMT_AmericanTower_ret_1d", "MSTR_Bitcoin3_ret_20d", "PCAR_PaccarInc_ret_5d", "Brent_Oil_FRED_ret_5d", "EWM_Malaysia_ret_1d", "DHR_vol_20d", "DE_Deere_vol_20d", "EWL_Switzerland_zscore_60d", "PAYX_Paychex_zscore_60d", "AMGN_Amgen_ret_1d", "US6M_Rate_ret_20d", "VRP_ma5"], "is_new": true}, {"model_id": "new_h5_STRESS_LightGBM_N25_t2", "algo": "LightGBM", "regime": "STRESS", "horizon": 5, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "MS_MorganStanley_ret_5d", "IBEX_Spain_ret_20d", "HD_zscore_60d", "EWM_Malaysia_vol_20d", "XLB_Materials_zscore_60d", "AMD_ret_1d", "CI_Cigna_vol_20d", "IYM_BasicMaterials_ret_20d", "spx_momentum_3d", "EWL_Switzerland_zscore_60d", "US5Y_Rate_ret_5d", "IYR_US_REIT2_zscore_60d", "US30Y_Rate_ret_20d", "EWM_Malaysia_ret_1d", "EWA_Australia_ret_1d", "DE_Deere_vol_20d", "NWL_Newell_ret_20d", "LOW_Lowes_ret_5d", "AVB_AvalonBay_zscore_60d", "AMD_ret_5d", "Core_PCE_zscore_60d", "EOG_EOGResources_vol_20d", "TED_Spread_zscore_60d"], "is_new": true}, {"model_id": "new_h5_STRESS_LightGBM_N25_t3", "algo": "LightGBM", "regime": "STRESS", "horizon": 5, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "HangSeng_HK_ret_1d", "IYR_US_REIT2_zscore_60d", "WTI_Oil_FRED_zscore_60d", "BLK_BlackRock_zscore_60d", "EWY_Korea_ret_20d", "BA_ret_1d", "BDX_Becton_Dickinson_ret_20d", "MS_MorganStanley_ret_1d", "PLD_Prologis_ret_5d", "US5Y_Rate_ret_5d", "VRP_ma5", "BTI_BritishAmerican_ret_5d", "EWG_Germany_ret_20d", "HD_ret_20d", "NEE_NextEra_ret_20d", "US30Y_Rate_ret_20d", "AVB_AvalonBay_zscore_60d", "MSTR_Bitcoin3_ret_20d", "AORD_AUS_zscore_60d", "MSTR_Bitcoin3_ret_5d", "LLY_zscore_60d", "MS_MorganStanley_zscore_60d", "BTI_BritishAmerican_ret_20d"], "is_new": true}, {"model_id": "new_h5_STRESS_LightGBM_N25_t4", "algo": "LightGBM", "regime": "STRESS", "horizon": 5, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "PPL_PPL_ret_1d", "DHR_ret_1d", "NFCI_ret_5d", "SBUX_vol_20d", "EQR_Equity_ret_1d", "EWJ_Japan_vol_20d", "HangSeng_HK_ret_5d", "ASX_Australia_vol_20d", "ORCL_vol_20d", "TM_Telephone_vol_20d", "TXN_vol_20d", "ORCL_zscore_60d", "SJM_JM_Smucker_ret_1d", "DOW_Price_zscore_60d", "SBUX_ret_5d", "Industrial_Production_zscore_60d", "EWC_Canada_zscore_60d", "T_ret_1d", "NOC_Northrop_ret_20d", "MS_MorganStanley_ret_1d", "XLB_Materials_zscore_60d", "spx_momentum_3d", "BDX_Becton_Dickinson_ret_20d"], "is_new": true}, {"model_id": "new_h5_STRESS_LightGBM_N25_t5", "algo": "LightGBM", "regime": "STRESS", "horizon": 5, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "CPB_CampbellSoup_vol_20d", "TM_Telephone_ret_1d", "AMGN_Amgen_ret_1d", "SLB_Schlumberger_ret_5d", "XLV_Health_zscore_60d", "EWG_Germany_vol_20d", "CPB_CampbellSoup_zscore_60d", "MS_MorganStanley_zscore_60d", "XLB_Materials_zscore_60d", "SLB_Schlumberger_ret_1d", "DE_Deere_vol_20d", "EWG_Germany_ret_20d", "SJM_JM_Smucker_ret_1d", "US5Y_Rate_ret_5d", "VRP_ma5", "MRK_Merck_zscore_60d", "EWS_Singapore_ret_5d", "HangSeng_HK_ret_5d", "gjr_condvar_h1", "AORD_AUS_zscore_60d", "IYM_BasicMaterials_ret_20d", "NFCI_ret_5d", "Michigan_Sentiment_ret_20d"], "is_new": true}, {"model_id": "new_h5_STRESS_LightGBM_N25_t6", "algo": "LightGBM", "regime": "STRESS", "horizon": 5, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "LOW_Lowes_ret_20d", "EOG_EOGResources_ret_5d", "MS_MorganStanley_zscore_60d", "PAYX_Paychex_zscore_60d", "INTC_ret_5d", "MS_MorganStanley_ret_1d", "LMT_LockheedMartin_vol_20d", "EQIX_Equinix_ret_5d", "XOM_ret_1d", "TM_Telephone_vol_20d", "3M_vol_20d", "MRK_Merck_zscore_60d", "EWS_Singapore_ret_5d", "ASX_Australia_vol_20d", "EWA_Australia_zscore_60d", "spx_vol_5d", "Core_PCE_zscore_60d", "AMD_ret_1d", "AMGN_Amgen_ret_1d", "GD_GeneralDynamics_zscore_60d", "ORCL_vol_20d", "AMT_AmericanTower_ret_1d", "IYR_US_REIT2_zscore_60d"], "is_new": true}, {"model_id": "new_h5_STRESS_LightGBM_N25_t7", "algo": "LightGBM", "regime": "STRESS", "horizon": 5, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "US3M_Rate_vol_20d", "XOM_ret_1d", "JNJ_ret_1d", "3M_ret_5d", "AXP_Amex_ret_20d", "vix_acceleration_1d", "US1Y_Rate_ret_5d", "PPL_PPL_ret_1d", "EOG_EOGResources_ret_5d", "MS_MorganStanley_zscore_60d", "GE_ret_1d", "NVDA_vol_20d", "DHR_vol_20d", "PAYX_Paychex_zscore_60d", "CPB_CampbellSoup_vol_20d", "NOC_Northrop_ret_20d", "PCAR_PaccarInc_ret_5d", "LMT_LockheedMartin_ret_1d", "DE_Deere_ret_5d", "AMD_ret_1d", "LOW_Lowes_ret_5d", "US3Y_Rate_ret_5d", "EWS_Singapore_ret_5d"], "is_new": true}, {"model_id": "new_h5_STRESS_LightGBM_N30_t0", "algo": "LightGBM", "regime": "STRESS", "horizon": 5, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "MO_AltriaMG_ret_1d", "SJM_JM_Smucker_ret_1d", "EFFR_vol_20d", "EWQ_France_ret_20d", "US6M_Rate_ret_20d", "SCHW_Schwab_ret_5d", "EXC_Exelon_zscore_60d", "ES_Evergy_ret_1d", "EWY_Korea_zscore_60d", "PG_ret_20d", "HD_ret_20d", "DOW_Price_zscore_60d", "IWM_SmallCap_vol_20d", "TM_Telephone_vol_20d", "VOD_Vodafone_zscore_60d", "CPB_CampbellSoup_ret_20d", "spx_abs_ret_max_5d", "INTC_ret_1d", "SBUX_zscore_60d", "IBEX_Spain_ret_20d", "Nikkei_Japan_vol_20d", "XOM_ret_1d", "CLX_Clorox_vol_20d", "US5Y_Rate_ret_5d", "TGT_Target_zscore_60d", "HUM_Humana_ret_5d", "NWL_Newell_ret_20d", "MRK_Merck_zscore_60d"], "is_new": true}, {"model_id": "new_h5_STRESS_LightGBM_N30_t1", "algo": "LightGBM", "regime": "STRESS", "horizon": 5, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "AORD_AUS_zscore_60d", "EWG_Germany_vol_20d", "ES_Evergy_ret_1d", "heston_var_ev_h5", "US30Y_Rate_ret_20d", "US1Y_Rate_ret_20d", "SLB_Schlumberger_ret_5d", "VOD_Vodafone_zscore_60d", "EXC_Exelon_zscore_60d", "TGT_Target_zscore_60d", "US3Y_Rate_ret_5d", "SBUX_ret_5d", "US1Y_Rate_ret_5d", "XLK_Tech_zscore_60d", "HangSeng_HK_ret_5d", "CPB_CampbellSoup_vol_20d", "SCHW_Schwab_ret_5d", "DAX_Germany_zscore_60d", "Nikkei_Japan_zscore_60d", "EWM_Malaysia_ret_1d", "LOW_Lowes_ret_20d", "DE_Deere_vol_20d", "SPY_zscore_60d", "hmm_p_stress", "DHR_ret_1d", "SJM_JM_Smucker_ret_5d", "AMD_ret_1d", "TED_Spread_zscore_60d"], "is_new": true}, {"model_id": "new_h5_STRESS_LightGBM_N30_t2", "algo": "LightGBM", "regime": "STRESS", "horizon": 5, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWA_Australia_zscore_60d", "spx_vol_5d", "ASX_Australia_vol_20d", "DOW_Price_zscore_60d", "LMT_LockheedMartin_ret_1d", "spx_momentum_3d", "AVB_AvalonBay_zscore_60d", "AMD_ret_1d", "DAX_Germany_zscore_60d", "US5Y_Rate_ret_5d", "SPY_zscore_60d", "XOM_ret_1d", "AORD_AUS_zscore_60d", "HD_zscore_60d", "EMR_Emerson_ret_20d", "NOC_Northrop_ret_20d", "US3Y_Rate_ret_5d", "T_ret_1d", "AXP_Amex_ret_20d", "TM_Telephone_vol_20d", "Retail_Sales_zscore_60d", "TXN_vol_20d", "EOG_EOGResources_vol_20d", "INTC_ret_1d", "EWQ_France_ret_20d", "EXC_Exelon_zscore_60d", "MSTR_Bitcoin3_ret_1d", "SCHW_Schwab_ret_5d"], "is_new": true}, {"model_id": "new_h5_STRESS_LightGBM_N30_t3", "algo": "LightGBM", "regime": "STRESS", "horizon": 5, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "hmm_p_stress", "ASX_Australia_vol_20d", "ORCL_vol_20d", "Brent_Oil_FRED_ret_20d", "MSTR_Bitcoin3_ret_5d", "BTI_BritishAmerican_ret_5d", "EWY_Korea_ret_20d", "Core_PCE_zscore_60d", "DOW_Price_zscore_60d", "DAX_Germany_vol_20d", "PFE_ret_1d", "TED_Spread_zscore_60d", "TGT_Target_zscore_60d", "XOM_ret_1d", "T10Y2Y_Spread_ret_5d", "CLX_Clorox_vol_20d", "spx_vol_5d", "ES_Evergy_ret_1d", "MS_MorganStanley_ret_5d", "NWL_Newell_ret_20d", "Michigan_Sentiment_ret_20d", "SLB_Schlumberger_ret_5d", "QQQ_vol_20d", "GD_GeneralDynamics_zscore_60d", "DHR_ret_1d", "EMR_Emerson_ret_20d", "EWQ_France_ret_20d", "EWM_Malaysia_vol_20d"], "is_new": true}, {"model_id": "new_h5_STRESS_LightGBM_N30_t4", "algo": "LightGBM", "regime": "STRESS", "horizon": 5, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "M_Macys_vol_20d", "BTI_BritishAmerican_ret_5d", "CMCSA_ret_1d", "PFE_ret_1d", "US3M_Rate_zscore_60d", "INTC_ret_5d", "HangSeng_HK_ret_5d", "VVIX_ret_20d", "MO_AltriaMG_ret_1d", "spx_momentum_3d", "vix_acceleration_1d", "CI_Cigna_vol_20d", "ORCL_zscore_60d", "AMD_ret_1d", "HUM_Humana_ret_5d", "EXC_Exelon_ret_1d", "IBEX_Spain_ret_20d", "Brent_Oil_FRED_ret_20d", "LUV_SouthwestAir_ret_5d", "EWA_Australia_zscore_60d", "MSTR_Bitcoin3_ret_5d", "DAX_Germany_vol_20d", "XLK_Tech_zscore_60d", "IYR_US_REIT2_zscore_60d", "XLB_Materials_zscore_60d", "TM_Telephone_vol_20d", "EWL_Switzerland_vol_20d", "US5Y_Rate_ret_5d"], "is_new": true}, {"model_id": "new_h5_STRESS_LightGBM_N30_t5", "algo": "LightGBM", "regime": "STRESS", "horizon": 5, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "MSTR_Bitcoin3_ret_1d", "QQQ_vol_20d", "AVB_AvalonBay_zscore_60d", "HangSeng_HK_ret_5d", "US7Y_Rate_ret_20d", "CPB_CampbellSoup_vol_20d", "LOW_Lowes_ret_5d", "MS_MorganStanley_ret_5d", "CPB_CampbellSoup_zscore_60d", "AMD_ret_1d", "BDX_Becton_Dickinson_ret_20d", "SBUX_vol_20d", "DAX_Germany_zscore_60d", "Nikkei_Japan_zscore_60d", "BLK_BlackRock_zscore_60d", "PLD_Prologis_ret_5d", "MSTR_Bitcoin3_ret_20d", "SO_SouthernCo_ret_5d", "LMT_LockheedMartin_vol_20d", "Brent_Oil_FRED_ret_20d", "NVDA_vol_20d", "HD_zscore_60d", "gjr_condvar_h1", "NWL_Newell_ret_20d", "HD_ret_5d", "ASX_Australia_ret_5d", "BA_ret_1d", "Industrial_Production_zscore_60d"], "is_new": true}, {"model_id": "new_h5_STRESS_LightGBM_N30_t6", "algo": "LightGBM", "regime": "STRESS", "horizon": 5, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "CPB_CampbellSoup_vol_20d", "DE_Deere_vol_20d", "SJM_JM_Smucker_ret_5d", "EOG_EOGResources_ret_5d", "SBUX_ret_5d", "DHR_ret_1d", "XOM_ret_1d", "EXC_Exelon_zscore_60d", "EFFR_vol_20d", "TM_Telephone_ret_1d", "XLY_Disc_vol_20d", "TXN_vol_20d", "Brent_Oil_FRED_ret_5d", "CTAS_Cintas_vol_20d", "NWL_Newell_ret_20d", "Nikkei_Japan_zscore_60d", "HD_ret_20d", "VVIX_ret_20d", "QQQ_vol_20d", "EMR_Emerson_ret_20d", "GE_ret_1d", "TED_Spread_vol_20d", "EWY_Korea_ret_20d", "EFFR_ret_1d", "INTC_ret_1d", "ITT_ITTInc_ret_5d", "SCHW_Schwab_ret_5d", "HangSeng_HK_ret_1d"], "is_new": true}, {"model_id": "new_h5_STRESS_LightGBM_N30_t7", "algo": "LightGBM", "regime": "STRESS", "horizon": 5, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWG_Germany_vol_20d", "SJM_JM_Smucker_ret_1d", "LMT_LockheedMartin_vol_20d", "US6M_Rate_ret_20d", "spx_momentum_3d", "EWC_Canada_zscore_60d", "EWM_Malaysia_ret_1d", "EXC_Exelon_ret_1d", "LUV_SouthwestAir_ret_5d", "US1Y_Rate_ret_20d", "ES_Evergy_ret_1d", "EWH_HongKong_ret_5d", "T_ret_1d", "TGT_Target_zscore_60d", "CLX_Clorox_vol_20d", "SJM_JM_Smucker_ret_5d", "SBUX_zscore_60d", "CTAS_Cintas_vol_20d", "MS_MorganStanley_zscore_60d", "VRP_ma5", "EWY_Korea_zscore_60d", "MSTR_Bitcoin3_ret_5d", "ORCL_zscore_60d", "LLY_zscore_60d", "Nikkei_Japan_zscore_60d", "US30Y_Rate_ret_20d", "DE_Deere_vol_20d", "EQR_Equity_ret_1d"], "is_new": true}, {"model_id": "new_h5_STRESS_GradientBoosting_N5_t0", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 5, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "PPL_PPL_ret_1d", "DE_Deere_vol_20d", "IYM_BasicMaterials_ret_20d"], "is_new": true}, {"model_id": "new_h5_STRESS_GradientBoosting_N5_t1", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 5, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWA_Australia_ret_1d", "CMCSA_ret_1d", "ITT_ITTInc_ret_5d"], "is_new": true}, {"model_id": "new_h5_STRESS_GradientBoosting_N5_t2", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 5, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "NWL_Newell_ret_20d", "SJM_JM_Smucker_ret_1d", "spx_abs_ret_max_5d"], "is_new": true}, {"model_id": "new_h5_STRESS_GradientBoosting_N5_t3", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 5, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "ORCL_vol_20d", "AMD_ret_5d", "Nikkei_Japan_zscore_60d"], "is_new": true}, {"model_id": "new_h5_STRESS_GradientBoosting_N5_t4", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 5, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "PLD_Prologis_ret_5d", "XOM_ret_20d", "EQR_Equity_ret_1d"], "is_new": true}, {"model_id": "new_h5_STRESS_GradientBoosting_N5_t5", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 5, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "ITT_ITTInc_ret_5d", "vix_acceleration_1d", "SBUX_vol_20d"], "is_new": true}, {"model_id": "new_h5_STRESS_GradientBoosting_N5_t6", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 5, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "CPB_CampbellSoup_vol_20d", "XOM_ret_1d", "HD_ret_5d"], "is_new": true}, {"model_id": "new_h5_STRESS_GradientBoosting_N5_t7", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 5, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "MS_MorganStanley_zscore_60d", "EWH_HongKong_ret_5d", "XLV_Health_zscore_60d"], "is_new": true}, {"model_id": "new_h5_STRESS_GradientBoosting_N8_t0", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 5, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "IBEX_Spain_ret_20d", "GE_ret_1d", "SCHW_Schwab_ret_5d", "Core_CPI_zscore_60d", "LUV_SouthwestAir_ret_5d", "US5Y_Rate_ret_5d"], "is_new": true}, {"model_id": "new_h5_STRESS_GradientBoosting_N8_t1", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 5, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "US5Y_Rate_ret_5d", "EXC_Exelon_zscore_60d", "gjr_condvar_h1", "GE_ret_1d", "IYM_BasicMaterials_ret_20d", "XOM_ret_1d"], "is_new": true}, {"model_id": "new_h5_STRESS_GradientBoosting_N8_t2", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 5, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "ASX_Australia_vol_20d", "XLV_Health_zscore_60d", "PCAR_PaccarInc_ret_5d", "IBEX_Spain_ret_20d", "CI_Cigna_vol_20d", "SLB_Schlumberger_ret_5d"], "is_new": true}, {"model_id": "new_h5_STRESS_GradientBoosting_N8_t3", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 5, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWY_Korea_zscore_60d", "AORD_AUS_zscore_60d", "IWM_SmallCap_vol_20d", "EWY_Korea_ret_20d", "LOW_Lowes_ret_5d", "DE_Deere_vol_20d"], "is_new": true}, {"model_id": "new_h5_STRESS_GradientBoosting_N8_t4", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 5, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "PG_ret_20d", "Michigan_Sentiment_ret_20d", "VOD_Vodafone_zscore_60d", "SBUX_vol_20d", "EWA_Australia_ret_1d", "spx_momentum_3d"], "is_new": true}, {"model_id": "new_h5_STRESS_GradientBoosting_N8_t5", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 5, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "Core_PCE_zscore_60d", "SLB_Schlumberger_ret_5d", "Nikkei_Japan_zscore_60d", "HD_ret_1d", "Industrial_Production_zscore_60d", "EWL_Switzerland_zscore_60d"], "is_new": true}, {"model_id": "new_h5_STRESS_GradientBoosting_N8_t6", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 5, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "XLY_Disc_vol_20d", "Brent_Oil_FRED_ret_5d", "PLD_Prologis_ret_5d", "EXC_Exelon_ret_1d", "US3Y_Rate_ret_5d", "US3M_Rate_vol_20d"], "is_new": true}, {"model_id": "new_h5_STRESS_GradientBoosting_N8_t7", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 5, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "ENB_EnbridgeInc_ret_1d", "PAYX_Paychex_vol_20d", "VRP_ma5", "CPB_CampbellSoup_ret_5d", "SPY_zscore_60d", "AMD_ret_1d"], "is_new": true}, {"model_id": "new_h5_STRESS_GradientBoosting_N10_t0", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 5, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "BA_ret_1d", "BDX_Becton_Dickinson_ret_20d", "NOC_Northrop_ret_20d", "US30Y_Rate_ret_20d", "TED_Spread_zscore_60d", "SBUX_vol_20d", "EWQ_France_ret_20d", "3M_vol_20d"], "is_new": true}, {"model_id": "new_h5_STRESS_GradientBoosting_N10_t1", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 5, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "AMZN_ret_5d", "LLY_zscore_60d", "NEE_NextEra_ret_20d", "heston_var_ev_h7", "EWY_Korea_ret_20d", "EWM_Malaysia_zscore_60d", "MS_MorganStanley_ret_5d", "EXC_Exelon_zscore_60d"], "is_new": true}, {"model_id": "new_h5_STRESS_GradientBoosting_N10_t2", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 5, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "TXN_vol_20d", "EMR_Emerson_ret_20d", "PAYX_Paychex_ret_20d", "CI_Cigna_vol_20d", "spx_vol_5d", "EFFR_vol_20d", "US1Y_Rate_ret_5d", "T_ret_1d"], "is_new": true}, {"model_id": "new_h5_STRESS_GradientBoosting_N10_t3", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 5, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "AMT_AmericanTower_ret_1d", "US3M_Rate_zscore_60d", "Michigan_Sentiment_ret_20d", "TXN_vol_20d", "PCAR_PaccarInc_ret_5d", "EOG_EOGResources_vol_20d", "MRK_Merck_zscore_60d", "HangSeng_HK_ret_1d"], "is_new": true}, {"model_id": "new_h5_STRESS_GradientBoosting_N10_t4", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 5, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "TM_Telephone_ret_1d", "TED_Spread_zscore_60d", "Brent_Oil_FRED_ret_20d", "NOC_Northrop_ret_20d", "SLB_Schlumberger_ret_1d", "DOW_Price_zscore_60d", "AMD_ret_1d", "HD_ret_1d"], "is_new": true}, {"model_id": "new_h5_STRESS_GradientBoosting_N10_t5", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 5, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "PG_ret_20d", "SBUX_vol_20d", "EWL_Switzerland_vol_20d", "CTAS_Cintas_vol_20d", "PCAR_PaccarInc_ret_5d", "HUM_Humana_ret_5d", "EOG_EOGResources_ret_5d", "SBUX_ret_5d"], "is_new": true}, {"model_id": "new_h5_STRESS_GradientBoosting_N10_t6", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 5, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "TXN_vol_20d", "US1Y_Rate_ret_5d", "PPL_PPL_ret_1d", "EFFR_vol_20d", "GE_ret_1d", "DHR_ret_1d", "PG_ret_20d", "LMT_LockheedMartin_ret_1d"], "is_new": true}, {"model_id": "new_h5_STRESS_GradientBoosting_N10_t7", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 5, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "MSTR_Bitcoin3_ret_20d", "IWM_SmallCap_vol_20d", "BDX_Becton_Dickinson_ret_20d", "IYM_BasicMaterials_ret_20d", "TXN_vol_20d", "AMD_ret_1d", "US3Y_Rate_ret_5d", "heston_var_ev_h3"], "is_new": true}, {"model_id": "new_h5_STRESS_GradientBoosting_N12_t0", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 5, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "HangSeng_HK_ret_1d", "ORCL_zscore_60d", "AXP_Amex_vol_20d", "MRK_Merck_zscore_60d", "XLK_Tech_zscore_60d", "heston_ev_h3", "FedFunds_zscore_60d", "T_ret_1d", "US1Y_Rate_ret_5d", "PLD_Prologis_ret_5d"], "is_new": true}, {"model_id": "new_h5_STRESS_GradientBoosting_N12_t1", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 5, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "SBUX_ret_5d", "heston_var_ev_h7", "gjr_condvar_h1", "XOM_ret_20d", "EWL_Switzerland_vol_20d", "EQR_Equity_ret_1d", "TM_Telephone_ret_1d", "heston_ev_h3", "HUM_Humana_ret_5d", "LLY_zscore_60d"], "is_new": true}, {"model_id": "new_h5_STRESS_GradientBoosting_N12_t2", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 5, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "TED_Spread_vol_20d", "BTI_BritishAmerican_ret_20d", "heston_var_ev_h3", "Retail_Sales_zscore_60d", "CLX_Clorox_vol_20d", "ASX_Australia_vol_20d", "T_ret_1d", "ES_Evergy_ret_1d", "DHR_vol_20d", "TXN_vol_20d"], "is_new": true}, {"model_id": "new_h5_STRESS_GradientBoosting_N12_t3", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 5, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWG_Germany_ret_20d", "EXC_Exelon_ret_1d", "US3M_Rate_zscore_60d", "Retail_Sales_zscore_60d", "PFE_ret_1d", "LOW_Lowes_ret_20d", "heston_var_ev_h3", "gjr_condvar_h1", "XLV_Health_zscore_60d", "EWY_Korea_ret_20d"], "is_new": true}, {"model_id": "new_h5_STRESS_GradientBoosting_N12_t4", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 5, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "LOW_Lowes_ret_20d", "SO_SouthernCo_ret_5d", "US1Y_Rate_ret_20d", "HD_ret_20d", "JNJ_ret_1d", "GILD_Gilead_ret_20d", "US6M_Rate_ret_20d", "ASX_Australia_ret_5d", "TXN_vol_20d", "NEE_NextEra_ret_20d"], "is_new": true}, {"model_id": "new_h5_STRESS_GradientBoosting_N12_t5", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 5, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWM_Malaysia_zscore_60d", "CPB_CampbellSoup_vol_20d", "CPB_CampbellSoup_ret_5d", "XLK_Tech_zscore_60d", "NEE_NextEra_ret_20d", "SJM_JM_Smucker_ret_1d", "MS_MorganStanley_zscore_60d", "vix_mean_abs_ret_5d", "HD_ret_1d", "PAYX_Paychex_zscore_60d"], "is_new": true}, {"model_id": "new_h5_STRESS_GradientBoosting_N12_t6", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 5, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "MSTR_Bitcoin3_ret_1d", "MSTR_Bitcoin3_ret_20d", "EWY_Korea_ret_20d", "SPY_zscore_60d", "vix_mean_abs_ret_5d", "ENB_EnbridgeInc_ret_1d", "SLB_Schlumberger_ret_5d", "EQR_Equity_ret_1d", "PAYX_Paychex_vol_20d", "AMD_ret_1d"], "is_new": true}, {"model_id": "new_h5_STRESS_GradientBoosting_N12_t7", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 5, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "SBUX_vol_20d", "EWQ_France_ret_20d", "ASX_Australia_ret_5d", "TGT_Target_zscore_60d", "IBEX_Spain_ret_20d", "EQR_Equity_ret_1d", "US6M_Rate_ret_20d", "SLB_Schlumberger_ret_1d", "INTC_ret_1d", "BLK_BlackRock_zscore_60d"], "is_new": true}, {"model_id": "new_h5_STRESS_GradientBoosting_N15_t0", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 5, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWJ_Japan_vol_20d", "Nikkei_Japan_zscore_60d", "TM_Telephone_vol_20d", "CLX_Clorox_vol_20d", "BA_ret_1d", "vix_acceleration_1d", "EXC_Exelon_zscore_60d", "NWL_Newell_ret_20d", "EWM_Malaysia_zscore_60d", "MS_MorganStanley_ret_1d", "EWS_Singapore_ret_5d", "DHR_vol_20d", "AMGN_Amgen_ret_1d"], "is_new": true}, {"model_id": "new_h5_STRESS_GradientBoosting_N15_t1", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 5, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "Brent_Oil_FRED_ret_20d", "DE_Deere_ret_5d", "EXC_Exelon_ret_1d", "VOD_Vodafone_zscore_60d", "CMCSA_ret_1d", "EOG_EOGResources_vol_20d", "heston_var_ev_h3", "EQR_Equity_ret_1d", "TM_Telephone_ret_1d", "spx_abs_ret_max_5d", "heston_ev_h3", "NOC_Northrop_ret_20d", "PAYX_Paychex_vol_20d"], "is_new": true}, {"model_id": "new_h5_STRESS_GradientBoosting_N15_t2", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 5, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "heston_var_ev_h3", "spx_abs_ret_max_5d", "US5Y_Rate_ret_5d", "EXC_Exelon_ret_1d", "US3Y_Rate_ret_5d", "DOW_Price_zscore_60d", "heston_var_ev_h7", "NOC_Northrop_ret_20d", "INTC_ret_5d", "CPB_CampbellSoup_vol_20d", "Core_PCE_zscore_60d", "TGT_Target_zscore_60d", "IYM_BasicMaterials_ret_20d"], "is_new": true}, {"model_id": "new_h5_STRESS_GradientBoosting_N15_t3", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 5, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EQR_Equity_ret_1d", "hmm_p_stress", "EWJ_Japan_vol_20d", "CMCSA_ret_1d", "ITT_ITTInc_ret_5d", "gjr_condvar_h1", "heston_var_ev_h7", "ENB_EnbridgeInc_ret_1d", "CPB_CampbellSoup_ret_5d", "vix_acceleration_1d", "heston_var_ev_h5", "TXN_vol_20d", "AMD_ret_5d"], "is_new": true}, {"model_id": "new_h5_STRESS_GradientBoosting_N15_t4", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 5, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "IBEX_Spain_ret_20d", "XLK_Tech_zscore_60d", "EFFR_ret_1d", "BLK_BlackRock_zscore_60d", "CTAS_Cintas_vol_20d", "T_ret_1d", "HangSeng_HK_ret_1d", "EWM_Malaysia_zscore_60d", "IWM_SmallCap_vol_20d", "VVIX_ret_20d", "QQQ_vol_20d", "EWM_Malaysia_ret_1d", "AMD_ret_5d"], "is_new": true}, {"model_id": "new_h5_STRESS_GradientBoosting_N15_t5", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 5, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "PPL_PPL_ret_1d", "DIS_vol_20d", "SBUX_ret_5d", "EWA_Australia_ret_1d", "T10Y2Y_Spread_ret_5d", "AXP_Amex_vol_20d", "CPB_CampbellSoup_vol_20d", "NOC_Northrop_ret_20d", "Industrial_Production_zscore_60d", "US1Y_Rate_ret_20d", "HangSeng_HK_vol_20d", "LOW_Lowes_ret_5d", "NVDA_vol_20d"], "is_new": true}, {"model_id": "new_h5_STRESS_GradientBoosting_N15_t6", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 5, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "HD_ret_5d", "3M_vol_20d", "LOW_Lowes_ret_5d", "EWM_Malaysia_ret_1d", "Nikkei_Japan_zscore_60d", "3M_ret_5d", "M_Macys_vol_20d", "IWM_SmallCap_vol_20d", "NWL_Newell_ret_20d", "SLB_Schlumberger_ret_1d", "CPB_CampbellSoup_ret_5d", "ITT_ITTInc_ret_5d", "EXC_Exelon_zscore_60d"], "is_new": true}, {"model_id": "new_h5_STRESS_GradientBoosting_N15_t7", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 5, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "CPB_CampbellSoup_ret_20d", "PAYX_Paychex_zscore_60d", "PPL_PPL_ret_1d", "EQR_Equity_ret_1d", "heston_ev_h3", "US3M_Rate_vol_20d", "ORCL_vol_20d", "DE_Deere_ret_5d", "DOW_Price_zscore_60d", "US6M_Rate_ret_20d", "ES_Evergy_ret_1d", "EWM_Malaysia_vol_20d", "HD_ret_20d"], "is_new": true}, {"model_id": "new_h5_STRESS_GradientBoosting_N20_t0", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 5, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "TM_Telephone_vol_20d", "HangSeng_HK_ret_5d", "GE_ret_1d", "NOC_Northrop_ret_20d", "Brent_Oil_FRED_ret_5d", "spx_momentum_3d", "ORCL_zscore_60d", "EWA_Australia_ret_1d", "DHR_ret_1d", "PAYX_Paychex_zscore_60d", "HangSeng_HK_ret_1d", "EWS_Singapore_ret_5d", "AMZN_ret_5d", "PAYX_Paychex_ret_20d", "MS_MorganStanley_ret_5d", "vix_mean_abs_ret_5d", "gjr_condvar_h1", "spx_vol_5d"], "is_new": true}, {"model_id": "new_h5_STRESS_GradientBoosting_N20_t1", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 5, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "gjr_condvar_h1", "3M_vol_20d", "AMD_ret_5d", "CPB_CampbellSoup_ret_5d", "heston_ev_h3", "SPY_zscore_60d", "HangSeng_HK_ret_1d", "PG_ret_20d", "vix_mean_abs_ret_5d", "heston_var_ev_h7", "Nikkei_Japan_vol_20d", "MSTR_Bitcoin3_ret_5d", "US7Y_Rate_ret_20d", "WTI_Oil_FRED_zscore_60d", "LUV_SouthwestAir_ret_5d", "EWA_Australia_ret_1d", "LLY_zscore_60d", "US3Y_Rate_ret_5d"], "is_new": true}, {"model_id": "new_h5_STRESS_GradientBoosting_N20_t2", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 5, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "DIS_vol_20d", "GILD_Gilead_ret_20d", "BA_ret_1d", "T_ret_1d", "EWY_Korea_zscore_60d", "vix_mean_abs_ret_5d", "QQQ_vol_20d", "MSTR_Bitcoin3_ret_20d", "LOW_Lowes_ret_20d", "EFFR_ret_1d", "AORD_AUS_zscore_60d", "LLY_zscore_60d", "EOG_EOGResources_vol_20d", "ES_Evergy_ret_1d", "HangSeng_HK_ret_1d", "PCAR_PaccarInc_ret_5d", "DAX_Germany_vol_20d", "TGT_Target_zscore_60d"], "is_new": true}, {"model_id": "new_h5_STRESS_GradientBoosting_N20_t3", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 5, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "gjr_condvar_h1", "EWQ_France_zscore_60d", "PCAR_PaccarInc_ret_5d", "CPB_CampbellSoup_vol_20d", "BLK_BlackRock_zscore_60d", "HD_ret_5d", "INTC_ret_5d", "heston_var_ev_h5", "T10Y2Y_Spread_ret_5d", "ES_Evergy_ret_1d", "MSTR_Bitcoin3_ret_20d", "VRP_ma5", "M_Macys_vol_20d", "ASX_Australia_vol_20d", "AMZN_ret_5d", "INTC_ret_1d", "BDX_Becton_Dickinson_ret_20d", "PFE_ret_1d"], "is_new": true}, {"model_id": "new_h5_STRESS_GradientBoosting_N20_t4", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 5, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EMR_Emerson_ret_20d", "Brent_Oil_FRED_ret_20d", "PCAR_PaccarInc_ret_5d", "SLB_Schlumberger_ret_1d", "AMD_ret_1d", "NFCI_ret_5d", "NEE_NextEra_ret_20d", "SCHW_Schwab_ret_5d", "PAYX_Paychex_ret_20d", "HangSeng_HK_ret_5d", "SJM_JM_Smucker_ret_5d", "BTI_BritishAmerican_ret_5d", "SLB_Schlumberger_ret_5d", "US6M_Rate_ret_20d", "EWG_Germany_ret_20d", "XOM_ret_20d", "M_Macys_vol_20d", "vix_mean_abs_ret_5d"], "is_new": true}, {"model_id": "new_h5_STRESS_GradientBoosting_N20_t5", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 5, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWQ_France_zscore_60d", "EWJ_Japan_vol_20d", "ORCL_vol_20d", "CCI_CrownCastle_vol_20d", "T_ret_1d", "VOD_Vodafone_zscore_60d", "CPB_CampbellSoup_ret_20d", "EWQ_France_ret_20d", "INTC_ret_5d", "JNJ_ret_1d", "VVIX_ret_20d", "LUV_SouthwestAir_ret_5d", "PFE_ret_1d", "EMR_Emerson_ret_20d", "PLD_Prologis_ret_5d", "DIS_vol_20d", "SCHW_Schwab_ret_5d", "US1Y_Rate_ret_20d"], "is_new": true}, {"model_id": "new_h5_STRESS_GradientBoosting_N20_t6", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 5, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "PG_ret_20d", "MSTR_Bitcoin3_ret_20d", "ES_Evergy_ret_1d", "EWQ_France_ret_20d", "SLB_Schlumberger_ret_5d", "ASX_Australia_vol_20d", "EFFR_vol_20d", "MS_MorganStanley_ret_5d", "SBUX_zscore_60d", "BDX_Becton_Dickinson_ret_20d", "VVIX_ret_20d", "HD_zscore_60d", "NEE_NextEra_ret_20d", "BTI_BritishAmerican_ret_5d", "US1Y_Rate_ret_5d", "Nikkei_Japan_vol_20d", "SBUX_ret_5d", "CLX_Clorox_vol_20d"], "is_new": true}, {"model_id": "new_h5_STRESS_GradientBoosting_N20_t7", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 5, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "SPY_zscore_60d", "DAX_Germany_vol_20d", "FedFunds_zscore_60d", "NVDA_vol_20d", "BA_ret_1d", "EXC_Exelon_ret_1d", "spx_vol_5d", "US5Y_Rate_ret_5d", "US6M_Rate_ret_20d", "CLX_Clorox_vol_20d", "DOW_Price_zscore_60d", "VOD_Vodafone_zscore_60d", "EWL_Switzerland_zscore_60d", "Nikkei_Japan_vol_20d", "EWG_Germany_ret_20d", "DIS_vol_20d", "AORD_AUS_zscore_60d", "ENB_EnbridgeInc_ret_1d"], "is_new": true}, {"model_id": "new_h5_STRESS_GradientBoosting_N25_t0", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 5, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWJ_Japan_vol_20d", "HangSeng_HK_ret_1d", "SBUX_vol_20d", "DOW_Price_zscore_60d", "AVB_AvalonBay_zscore_60d", "US3M_Rate_vol_20d", "ORCL_vol_20d", "SBUX_zscore_60d", "EWS_Singapore_ret_5d", "TM_Telephone_ret_1d", "NWL_Newell_ret_20d", "MS_MorganStanley_ret_1d", "SBUX_ret_5d", "LLY_zscore_60d", "BTI_BritishAmerican_ret_5d", "Nikkei_Japan_zscore_60d", "MRK_Merck_zscore_60d", "EWH_HongKong_ret_5d", "PAYX_Paychex_ret_20d", "XOM_ret_20d", "PFE_ret_1d", "AMZN_ret_5d", "heston_var_ev_h3"], "is_new": true}, {"model_id": "new_h5_STRESS_GradientBoosting_N25_t1", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 5, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "XLY_Disc_vol_20d", "VVIX_ret_20d", "CI_Cigna_vol_20d", "HangSeng_HK_ret_1d", "EWM_Malaysia_zscore_60d", "TXN_vol_20d", "EWQ_France_ret_20d", "XLV_Health_zscore_60d", "PAYX_Paychex_ret_20d", "INTC_ret_5d", "3M_ret_5d", "JNJ_ret_1d", "EWL_Switzerland_zscore_60d", "AMD_ret_5d", "T_ret_1d", "EWA_Australia_zscore_60d", "EWJ_Japan_vol_20d", "MSTR_Bitcoin3_ret_1d", "ASX_Australia_vol_20d", "CPB_CampbellSoup_vol_20d", "HUM_Humana_ret_5d", "Nikkei_Japan_vol_20d", "SBUX_zscore_60d"], "is_new": true}, {"model_id": "new_h5_STRESS_GradientBoosting_N25_t2", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 5, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "heston_ev_h3", "TM_Telephone_vol_20d", "US30Y_Rate_ret_20d", "SPY_zscore_60d", "US1Y_Rate_ret_5d", "ES_Evergy_ret_1d", "SBUX_vol_20d", "BA_ret_1d", "EWQ_France_zscore_60d", "EFFR_vol_20d", "EOG_EOGResources_ret_5d", "HUM_Humana_ret_5d", "SLB_Schlumberger_ret_1d", "XLK_Tech_zscore_60d", "BTI_BritishAmerican_ret_20d", "EWQ_France_ret_20d", "MSTR_Bitcoin3_ret_1d", "ASX_Australia_vol_20d", "HangSeng_HK_vol_20d", "EWA_Australia_zscore_60d", "PAYX_Paychex_vol_20d", "GD_GeneralDynamics_zscore_60d", "LLY_zscore_60d"], "is_new": true}, {"model_id": "new_h5_STRESS_GradientBoosting_N25_t3", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 5, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWY_Korea_ret_20d", "US3M_Rate_zscore_60d", "HangSeng_HK_ret_5d", "AMGN_Amgen_ret_1d", "TM_Telephone_ret_1d", "CI_Cigna_vol_20d", "US3Y_Rate_ret_5d", "DE_Deere_ret_5d", "SJM_JM_Smucker_ret_1d", "IYM_BasicMaterials_ret_20d", "SLB_Schlumberger_ret_1d", "GILD_Gilead_ret_20d", "T10Y2Y_Spread_ret_5d", "EFFR_vol_20d", "SBUX_zscore_60d", "ORCL_zscore_60d", "MO_AltriaMG_ret_1d", "VRP_ma5", "spx_vol_5d", "PAYX_Paychex_zscore_60d", "ES_Evergy_ret_1d", "EOG_EOGResources_ret_5d", "PG_ret_20d"], "is_new": true}, {"model_id": "new_h5_STRESS_GradientBoosting_N25_t4", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 5, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EXC_Exelon_ret_1d", "spx_abs_ret_max_5d", "MSTR_Bitcoin3_ret_1d", "PG_ret_20d", "LMT_LockheedMartin_vol_20d", "XLK_Tech_zscore_60d", "INTC_ret_1d", "SBUX_ret_5d", "XLV_Health_zscore_60d", "gjr_condvar_h1", "MSTR_Bitcoin3_ret_5d", "EWG_Germany_vol_20d", "EWS_Singapore_ret_5d", "QQQ_vol_20d", "Core_PCE_zscore_60d", "Brent_Oil_FRED_ret_20d", "WTI_Oil_FRED_zscore_60d", "EWG_Germany_ret_20d", "INTC_ret_5d", "heston_var_ev_h3", "spx_momentum_3d", "AORD_AUS_zscore_60d", "US1Y_Rate_ret_20d"], "is_new": true}, {"model_id": "new_h5_STRESS_GradientBoosting_N25_t5", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 5, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "BA_ret_1d", "TED_Spread_zscore_60d", "CTAS_Cintas_vol_20d", "PLD_Prologis_ret_5d", "AXP_Amex_vol_20d", "EWG_Germany_vol_20d", "ORCL_zscore_60d", "MS_MorganStanley_zscore_60d", "BTI_BritishAmerican_ret_5d", "EWL_Switzerland_vol_20d", "IWM_SmallCap_vol_20d", "EMR_Emerson_ret_20d", "HD_ret_5d", "EFFR_ret_1d", "US1Y_Rate_ret_20d", "CPB_CampbellSoup_vol_20d", "Nikkei_Japan_vol_20d", "US6M_Rate_ret_20d", "GILD_Gilead_ret_20d", "IYM_BasicMaterials_ret_20d", "EWQ_France_zscore_60d", "TXN_vol_20d", "EXC_Exelon_zscore_60d"], "is_new": true}, {"model_id": "new_h5_STRESS_GradientBoosting_N25_t6", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 5, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "heston_ev_h3", "vix_mean_abs_ret_5d", "heston_var_ev_h7", "PG_ret_20d", "DE_Deere_vol_20d", "HD_ret_1d", "GE_ret_1d", "heston_var_ev_h3", "ORCL_zscore_60d", "LLY_zscore_60d", "EWG_Germany_ret_20d", "XLY_Disc_vol_20d", "US1Y_Rate_ret_20d", "PAYX_Paychex_vol_20d", "US7Y_Rate_ret_20d", "spx_abs_ret_max_5d", "ASX_Australia_ret_5d", "CLX_Clorox_vol_20d", "HD_ret_20d", "SPY_zscore_60d", "DIS_vol_20d", "ENB_EnbridgeInc_ret_1d", "EWC_Canada_zscore_60d"], "is_new": true}, {"model_id": "new_h5_STRESS_GradientBoosting_N25_t7", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 5, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "TGT_Target_zscore_60d", "NFCI_ret_5d", "ITT_ITTInc_ret_5d", "AMD_ret_1d", "MSTR_Bitcoin3_ret_20d", "NOC_Northrop_ret_20d", "AMZN_ret_5d", "Core_PCE_zscore_60d", "CI_Cigna_vol_20d", "M_Macys_vol_20d", "LLY_zscore_60d", "Industrial_Production_zscore_60d", "TED_Spread_vol_20d", "heston_ev_h3", "LMT_LockheedMartin_vol_20d", "CPB_CampbellSoup_ret_20d", "CCI_CrownCastle_vol_20d", "gjr_condvar_h1", "AORD_AUS_zscore_60d", "EWQ_France_ret_20d", "spx_vol_5d", "LUV_SouthwestAir_ret_5d", "MO_AltriaMG_ret_1d"], "is_new": true}, {"model_id": "new_h5_STRESS_GradientBoosting_N30_t0", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 5, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "NOC_Northrop_ret_20d", "M_Macys_vol_20d", "spx_abs_ret_max_5d", "AORD_AUS_zscore_60d", "LOW_Lowes_ret_20d", "NWL_Newell_ret_20d", "SBUX_ret_5d", "Retail_Sales_zscore_60d", "ES_Evergy_ret_1d", "CPB_CampbellSoup_vol_20d", "EWA_Australia_ret_1d", "ORCL_vol_20d", "AMGN_Amgen_ret_1d", "AVB_AvalonBay_zscore_60d", "TM_Telephone_ret_1d", "GE_ret_1d", "EWJ_Japan_vol_20d", "TM_Telephone_vol_20d", "ASX_Australia_ret_5d", "AMZN_ret_5d", "hmm_p_stress", "EWS_Singapore_ret_5d", "EWM_Malaysia_zscore_60d", "SCHW_Schwab_ret_5d", "TED_Spread_zscore_60d", "DHR_ret_1d", "VOD_Vodafone_zscore_60d", "VRP_ma5"], "is_new": true}, {"model_id": "new_h5_STRESS_GradientBoosting_N30_t1", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 5, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "spx_vol_5d", "TED_Spread_vol_20d", "PAYX_Paychex_zscore_60d", "EOG_EOGResources_ret_5d", "BTI_BritishAmerican_ret_20d", "US5Y_Rate_ret_5d", "CPB_CampbellSoup_ret_5d", "MSTR_Bitcoin3_ret_5d", "EWC_Canada_zscore_60d", "3M_vol_20d", "XLV_Health_zscore_60d", "CTAS_Cintas_vol_20d", "Nikkei_Japan_zscore_60d", "IYM_BasicMaterials_ret_20d", "CMCSA_ret_1d", "NVDA_vol_20d", "TXN_vol_20d", "AVB_AvalonBay_zscore_60d", "AORD_AUS_zscore_60d", "GE_ret_1d", "SBUX_ret_5d", "ES_Evergy_ret_1d", "CPB_CampbellSoup_vol_20d", "IWM_SmallCap_vol_20d", "heston_var_ev_h3", "XLY_Disc_vol_20d", "PPL_PPL_ret_1d", "ASX_Australia_ret_5d"], "is_new": true}, {"model_id": "new_h5_STRESS_GradientBoosting_N30_t2", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 5, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "CCI_CrownCastle_vol_20d", "SBUX_vol_20d", "BLK_BlackRock_zscore_60d", "ES_Evergy_ret_1d", "AMD_ret_1d", "BA_ret_1d", "LMT_LockheedMartin_ret_1d", "IWM_SmallCap_vol_20d", "LUV_SouthwestAir_ret_5d", "AMD_ret_5d", "CPB_CampbellSoup_zscore_60d", "TXN_vol_20d", "PAYX_Paychex_zscore_60d", "TED_Spread_zscore_60d", "US6M_Rate_ret_20d", "heston_ev_h3", "BDX_Becton_Dickinson_ret_20d", "heston_var_ev_h5", "spx_momentum_3d", "VOD_Vodafone_zscore_60d", "FedFunds_zscore_60d", "DHR_vol_20d", "EWM_Malaysia_ret_1d", "DE_Deere_ret_5d", "LMT_LockheedMartin_vol_20d", "LLY_zscore_60d", "VRP_ma5", "EWG_Germany_ret_20d"], "is_new": true}, {"model_id": "new_h5_STRESS_GradientBoosting_N30_t3", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 5, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWH_HongKong_ret_5d", "EOG_EOGResources_vol_20d", "heston_var_ev_h7", "CI_Cigna_vol_20d", "vix_acceleration_1d", "HangSeng_HK_ret_5d", "EWC_Canada_zscore_60d", "3M_vol_20d", "DHR_vol_20d", "IYM_BasicMaterials_ret_20d", "SO_SouthernCo_ret_5d", "EWG_Germany_ret_20d", "EWY_Korea_ret_20d", "AMGN_Amgen_ret_1d", "EWL_Switzerland_vol_20d", "M_Macys_vol_20d", "Brent_Oil_FRED_ret_20d", "EWQ_France_zscore_60d", "DE_Deere_vol_20d", "DHR_ret_1d", "SBUX_zscore_60d", "NWL_Newell_ret_20d", "US6M_Rate_ret_20d", "AMD_ret_5d", "EXC_Exelon_ret_1d", "SBUX_ret_5d", "FedFunds_zscore_60d", "HangSeng_HK_ret_1d"], "is_new": true}, {"model_id": "new_h5_STRESS_GradientBoosting_N30_t4", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 5, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "Core_PCE_zscore_60d", "AORD_AUS_zscore_60d", "PAYX_Paychex_ret_20d", "heston_var_ev_h3", "US1Y_Rate_ret_5d", "AMD_ret_5d", "spx_abs_ret_max_5d", "SBUX_vol_20d", "NVDA_vol_20d", "MSTR_Bitcoin3_ret_20d", "SBUX_ret_5d", "TED_Spread_zscore_60d", "HangSeng_HK_vol_20d", "Industrial_Production_zscore_60d", "EWG_Germany_vol_20d", "EWC_Canada_zscore_60d", "HD_zscore_60d", "EWA_Australia_ret_1d", "EWS_Singapore_ret_5d", "LMT_LockheedMartin_ret_1d", "PFE_ret_1d", "SO_SouthernCo_ret_5d", "HangSeng_HK_ret_1d", "EWL_Switzerland_zscore_60d", "PCAR_PaccarInc_ret_5d", "AVB_AvalonBay_zscore_60d", "BTI_BritishAmerican_ret_5d", "Brent_Oil_FRED_ret_20d"], "is_new": true}, {"model_id": "new_h5_STRESS_GradientBoosting_N30_t5", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 5, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "M_Macys_vol_20d", "CI_Cigna_vol_20d", "EWG_Germany_vol_20d", "MS_MorganStanley_ret_5d", "EWJ_Japan_vol_20d", "SBUX_ret_5d", "IWM_SmallCap_vol_20d", "AMZN_ret_5d", "NVDA_vol_20d", "AMT_AmericanTower_ret_1d", "INTC_ret_1d", "BLK_BlackRock_zscore_60d", "EWL_Switzerland_zscore_60d", "EWS_Singapore_ret_5d", "MRK_Merck_zscore_60d", "CPB_CampbellSoup_ret_20d", "CTAS_Cintas_vol_20d", "MS_MorganStanley_zscore_60d", "EWQ_France_ret_20d", "Retail_Sales_zscore_60d", "US1Y_Rate_ret_5d", "HD_ret_5d", "LMT_LockheedMartin_vol_20d", "PAYX_Paychex_zscore_60d", "WTI_Oil_FRED_zscore_60d", "NFCI_ret_5d", "EOG_EOGResources_ret_5d", "LMT_LockheedMartin_ret_1d"], "is_new": true}, {"model_id": "new_h5_STRESS_GradientBoosting_N30_t6", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 5, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "spx_vol_5d", "hmm_p_stress", "DAX_Germany_zscore_60d", "Nikkei_Japan_zscore_60d", "3M_ret_5d", "EOG_EOGResources_ret_5d", "ORCL_zscore_60d", "EWQ_France_ret_20d", "ES_Evergy_ret_1d", "vix_mean_abs_ret_5d", "M_Macys_vol_20d", "XLV_Health_zscore_60d", "XLF_Fin_vol_20d", "DE_Deere_ret_5d", "PAYX_Paychex_zscore_60d", "DE_Deere_vol_20d", "SPY_zscore_60d", "PG_ret_20d", "AMZN_ret_5d", "XLY_Disc_vol_20d", "heston_var_ev_h3", "HangSeng_HK_ret_5d", "AMD_ret_5d", "AMGN_Amgen_ret_1d", "SBUX_zscore_60d", "IYM_BasicMaterials_ret_20d", "CCI_CrownCastle_vol_20d", "BTI_BritishAmerican_ret_5d"], "is_new": true}, {"model_id": "new_h5_STRESS_GradientBoosting_N30_t7", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 5, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "AMD_ret_1d", "US5Y_Rate_ret_5d", "vix_acceleration_1d", "NFCI_ret_5d", "LMT_LockheedMartin_vol_20d", "NVDA_vol_20d", "PLD_Prologis_ret_5d", "US1Y_Rate_ret_5d", "US3M_Rate_zscore_60d", "LOW_Lowes_ret_20d", "gjr_condvar_h1", "MSTR_Bitcoin3_ret_20d", "GILD_Gilead_ret_20d", "spx_abs_ret_max_5d", "XLF_Fin_vol_20d", "TED_Spread_vol_20d", "CMCSA_ret_1d", "PAYX_Paychex_ret_20d", "BTI_BritishAmerican_ret_20d", "EWH_HongKong_ret_5d", "ASX_Australia_vol_20d", "EWL_Switzerland_zscore_60d", "EMR_Emerson_ret_20d", "DIS_vol_20d", "US30Y_Rate_ret_20d", "HD_zscore_60d", "HangSeng_HK_ret_1d", "EOG_EOGResources_vol_20d"], "is_new": true}, {"model_id": "new_h5_STRESS_RandomForest_N5_t0", "algo": "RandomForest", "regime": "STRESS", "horizon": 5, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "TXN_vol_20d", "HD_zscore_60d", "EWG_Germany_vol_20d"], "is_new": true}, {"model_id": "new_h5_STRESS_RandomForest_N5_t1", "algo": "RandomForest", "regime": "STRESS", "horizon": 5, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWM_Malaysia_vol_20d", "ASX_Australia_vol_20d", "BLK_BlackRock_zscore_60d"], "is_new": true}, {"model_id": "new_h5_STRESS_RandomForest_N5_t2", "algo": "RandomForest", "regime": "STRESS", "horizon": 5, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "XOM_ret_1d", "IYM_BasicMaterials_ret_20d", "MSTR_Bitcoin3_ret_1d"], "is_new": true}, {"model_id": "new_h5_STRESS_RandomForest_N5_t3", "algo": "RandomForest", "regime": "STRESS", "horizon": 5, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "VVIX_ret_20d", "EWC_Canada_zscore_60d", "3M_ret_5d"], "is_new": true}, {"model_id": "new_h5_STRESS_RandomForest_N5_t4", "algo": "RandomForest", "regime": "STRESS", "horizon": 5, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "AMZN_ret_5d", "EOG_EOGResources_ret_5d", "PAYX_Paychex_zscore_60d"], "is_new": true}, {"model_id": "new_h5_STRESS_RandomForest_N5_t5", "algo": "RandomForest", "regime": "STRESS", "horizon": 5, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EOG_EOGResources_vol_20d", "IWM_SmallCap_vol_20d", "LOW_Lowes_ret_20d"], "is_new": true}, {"model_id": "new_h5_STRESS_RandomForest_N5_t6", "algo": "RandomForest", "regime": "STRESS", "horizon": 5, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "3M_vol_20d", "ORCL_vol_20d", "EMR_Emerson_ret_20d"], "is_new": true}, {"model_id": "new_h5_STRESS_RandomForest_N5_t7", "algo": "RandomForest", "regime": "STRESS", "horizon": 5, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "TED_Spread_vol_20d", "Retail_Sales_zscore_60d", "HD_ret_20d"], "is_new": true}, {"model_id": "new_h5_STRESS_RandomForest_N8_t0", "algo": "RandomForest", "regime": "STRESS", "horizon": 5, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "3M_vol_20d", "LMT_LockheedMartin_ret_1d", "EWM_Malaysia_vol_20d", "LMT_LockheedMartin_vol_20d", "XLV_Health_zscore_60d", "NVDA_vol_20d"], "is_new": true}, {"model_id": "new_h5_STRESS_RandomForest_N8_t1", "algo": "RandomForest", "regime": "STRESS", "horizon": 5, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "AMD_ret_5d", "ASX_Australia_ret_5d", "VOD_Vodafone_zscore_60d", "SBUX_zscore_60d", "MSTR_Bitcoin3_ret_20d", "EWM_Malaysia_vol_20d"], "is_new": true}, {"model_id": "new_h5_STRESS_RandomForest_N8_t2", "algo": "RandomForest", "regime": "STRESS", "horizon": 5, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "PFE_ret_1d", "spx_momentum_3d", "EWS_Singapore_ret_5d", "XLF_Fin_vol_20d", "ORCL_zscore_60d", "MSTR_Bitcoin3_ret_5d"], "is_new": true}, {"model_id": "new_h5_STRESS_RandomForest_N8_t3", "algo": "RandomForest", "regime": "STRESS", "horizon": 5, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "US6M_Rate_ret_20d", "CPB_CampbellSoup_ret_20d", "GILD_Gilead_ret_20d", "DIS_vol_20d", "AMD_ret_1d", "EWM_Malaysia_zscore_60d"], "is_new": true}, {"model_id": "new_h5_STRESS_RandomForest_N8_t4", "algo": "RandomForest", "regime": "STRESS", "horizon": 5, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "BTI_BritishAmerican_ret_20d", "IBEX_Spain_ret_20d", "EWY_Korea_ret_20d", "BTI_BritishAmerican_ret_5d", "HangSeng_HK_ret_1d", "AORD_AUS_zscore_60d"], "is_new": true}, {"model_id": "new_h5_STRESS_RandomForest_N8_t5", "algo": "RandomForest", "regime": "STRESS", "horizon": 5, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EMR_Emerson_ret_20d", "XLK_Tech_zscore_60d", "MO_AltriaMG_ret_1d", "EWQ_France_zscore_60d", "EWG_Germany_ret_20d", "PG_ret_20d"], "is_new": true}, {"model_id": "new_h5_STRESS_RandomForest_N8_t6", "algo": "RandomForest", "regime": "STRESS", "horizon": 5, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "AMD_ret_5d", "TM_Telephone_ret_1d", "XOM_ret_20d", "TGT_Target_zscore_60d", "SLB_Schlumberger_ret_1d", "LUV_SouthwestAir_ret_5d"], "is_new": true}, {"model_id": "new_h5_STRESS_RandomForest_N8_t7", "algo": "RandomForest", "regime": "STRESS", "horizon": 5, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "spx_momentum_3d", "TXN_vol_20d", "IYR_US_REIT2_zscore_60d", "DOW_Price_zscore_60d", "BTI_BritishAmerican_ret_5d", "BA_ret_1d"], "is_new": true}, {"model_id": "new_h5_STRESS_RandomForest_N10_t0", "algo": "RandomForest", "regime": "STRESS", "horizon": 5, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "GD_GeneralDynamics_zscore_60d", "US1Y_Rate_ret_20d", "LOW_Lowes_ret_5d", "CMCSA_ret_1d", "MO_AltriaMG_ret_1d", "LMT_LockheedMartin_vol_20d", "US3M_Rate_zscore_60d", "Michigan_Sentiment_ret_20d"], "is_new": true}, {"model_id": "new_h5_STRESS_RandomForest_N10_t1", "algo": "RandomForest", "regime": "STRESS", "horizon": 5, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "DE_Deere_vol_20d", "Core_PCE_zscore_60d", "SPY_zscore_60d", "PG_ret_20d", "AMD_ret_5d", "EWQ_France_zscore_60d", "XLV_Health_zscore_60d", "LUV_SouthwestAir_ret_5d"], "is_new": true}, {"model_id": "new_h5_STRESS_RandomForest_N10_t2", "algo": "RandomForest", "regime": "STRESS", "horizon": 5, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWM_Malaysia_zscore_60d", "vix_acceleration_1d", "LOW_Lowes_ret_5d", "XLY_Disc_vol_20d", "FedFunds_zscore_60d", "AMD_ret_1d", "DE_Deere_vol_20d", "BTI_BritishAmerican_ret_5d"], "is_new": true}, {"model_id": "new_h5_STRESS_RandomForest_N10_t3", "algo": "RandomForest", "regime": "STRESS", "horizon": 5, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWM_Malaysia_vol_20d", "JNJ_ret_1d", "XOM_ret_20d", "MS_MorganStanley_ret_1d", "EWG_Germany_vol_20d", "MSTR_Bitcoin3_ret_5d", "DIS_vol_20d", "DHR_vol_20d"], "is_new": true}, {"model_id": "new_h5_STRESS_RandomForest_N10_t4", "algo": "RandomForest", "regime": "STRESS", "horizon": 5, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "Nikkei_Japan_vol_20d", "IYR_US_REIT2_zscore_60d", "LUV_SouthwestAir_ret_5d", "NFCI_ret_5d", "DAX_Germany_vol_20d", "vix_acceleration_1d", "SBUX_zscore_60d", "spx_momentum_3d"], "is_new": true}, {"model_id": "new_h5_STRESS_RandomForest_N10_t5", "algo": "RandomForest", "regime": "STRESS", "horizon": 5, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "spx_momentum_3d", "LLY_zscore_60d", "CI_Cigna_vol_20d", "Retail_Sales_zscore_60d", "DE_Deere_ret_5d", "PG_ret_20d", "HD_ret_1d", "T_ret_1d"], "is_new": true}, {"model_id": "new_h5_STRESS_RandomForest_N10_t6", "algo": "RandomForest", "regime": "STRESS", "horizon": 5, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "AVB_AvalonBay_zscore_60d", "LMT_LockheedMartin_vol_20d", "US6M_Rate_ret_20d", "WTI_Oil_FRED_zscore_60d", "FedFunds_zscore_60d", "HD_ret_1d", "MSTR_Bitcoin3_ret_1d", "CPB_CampbellSoup_ret_5d"], "is_new": true}, {"model_id": "new_h5_STRESS_RandomForest_N10_t7", "algo": "RandomForest", "regime": "STRESS", "horizon": 5, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "M_Macys_vol_20d", "BDX_Becton_Dickinson_ret_20d", "NVDA_vol_20d", "SJM_JM_Smucker_ret_5d", "HD_ret_5d", "HD_zscore_60d", "SJM_JM_Smucker_ret_1d", "3M_ret_5d"], "is_new": true}, {"model_id": "new_h5_STRESS_RandomForest_N12_t0", "algo": "RandomForest", "regime": "STRESS", "horizon": 5, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "XLF_Fin_vol_20d", "BTI_BritishAmerican_ret_20d", "EFFR_vol_20d", "MSTR_Bitcoin3_ret_5d", "AORD_AUS_zscore_60d", "EFFR_ret_1d", "heston_ev_h3", "SLB_Schlumberger_ret_1d", "PCAR_PaccarInc_ret_5d", "heston_var_ev_h3"], "is_new": true}, {"model_id": "new_h5_STRESS_RandomForest_N12_t1", "algo": "RandomForest", "regime": "STRESS", "horizon": 5, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "CPB_CampbellSoup_zscore_60d", "AMZN_ret_5d", "Nikkei_Japan_vol_20d", "HUM_Humana_ret_5d", "SLB_Schlumberger_ret_5d", "BA_ret_1d", "DIS_vol_20d", "EWA_Australia_zscore_60d", "LOW_Lowes_ret_20d", "MS_MorganStanley_zscore_60d"], "is_new": true}, {"model_id": "new_h5_STRESS_RandomForest_N12_t2", "algo": "RandomForest", "regime": "STRESS", "horizon": 5, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "HD_ret_20d", "TED_Spread_vol_20d", "HangSeng_HK_ret_1d", "TM_Telephone_vol_20d", "heston_ev_h3", "XLB_Materials_zscore_60d", "IBEX_Spain_ret_20d", "MS_MorganStanley_ret_1d", "IYM_BasicMaterials_ret_20d", "FedFunds_zscore_60d"], "is_new": true}, {"model_id": "new_h5_STRESS_RandomForest_N12_t3", "algo": "RandomForest", "regime": "STRESS", "horizon": 5, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EOG_EOGResources_vol_20d", "DHR_vol_20d", "WTI_Oil_FRED_zscore_60d", "NFCI_ret_5d", "BLK_BlackRock_zscore_60d", "NWL_Newell_ret_20d", "XLY_Disc_vol_20d", "CI_Cigna_vol_20d", "EWY_Korea_ret_20d", "US1Y_Rate_ret_5d"], "is_new": true}, {"model_id": "new_h5_STRESS_RandomForest_N12_t4", "algo": "RandomForest", "regime": "STRESS", "horizon": 5, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "XOM_ret_1d", "INTC_ret_1d", "DHR_vol_20d", "Brent_Oil_FRED_ret_5d", "US1Y_Rate_ret_5d", "NFCI_ret_5d", "Nikkei_Japan_zscore_60d", "PAYX_Paychex_ret_20d", "SPY_zscore_60d", "EWJ_Japan_vol_20d"], "is_new": true}, {"model_id": "new_h5_STRESS_RandomForest_N12_t5", "algo": "RandomForest", "regime": "STRESS", "horizon": 5, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "Nikkei_Japan_zscore_60d", "PCAR_PaccarInc_ret_5d", "NFCI_ret_5d", "MO_AltriaMG_ret_1d", "US7Y_Rate_ret_20d", "HD_zscore_60d", "US1Y_Rate_ret_5d", "MS_MorganStanley_zscore_60d", "EWM_Malaysia_ret_1d", "PAYX_Paychex_zscore_60d"], "is_new": true}, {"model_id": "new_h5_STRESS_RandomForest_N12_t6", "algo": "RandomForest", "regime": "STRESS", "horizon": 5, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "VOD_Vodafone_zscore_60d", "PAYX_Paychex_ret_20d", "CTAS_Cintas_vol_20d", "CCI_CrownCastle_vol_20d", "T10Y2Y_Spread_ret_5d", "XLY_Disc_vol_20d", "SBUX_ret_5d", "DE_Deere_ret_5d", "HD_ret_20d", "SPY_zscore_60d"], "is_new": true}, {"model_id": "new_h5_STRESS_RandomForest_N12_t7", "algo": "RandomForest", "regime": "STRESS", "horizon": 5, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWC_Canada_zscore_60d", "EWL_Switzerland_vol_20d", "Retail_Sales_zscore_60d", "gjr_condvar_h1", "TM_Telephone_ret_1d", "PCAR_PaccarInc_ret_5d", "EXC_Exelon_zscore_60d", "PAYX_Paychex_ret_20d", "EFFR_ret_1d", "NFCI_ret_5d"], "is_new": true}, {"model_id": "new_h5_STRESS_RandomForest_N15_t0", "algo": "RandomForest", "regime": "STRESS", "horizon": 5, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "SPY_zscore_60d", "CTAS_Cintas_vol_20d", "EWM_Malaysia_zscore_60d", "NOC_Northrop_ret_20d", "BTI_BritishAmerican_ret_20d", "Industrial_Production_zscore_60d", "spx_momentum_3d", "XLV_Health_zscore_60d", "EWL_Switzerland_zscore_60d", "M_Macys_vol_20d", "HangSeng_HK_vol_20d", "AXP_Amex_vol_20d", "SBUX_ret_5d"], "is_new": true}, {"model_id": "new_h5_STRESS_RandomForest_N15_t1", "algo": "RandomForest", "regime": "STRESS", "horizon": 5, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "gjr_condvar_h1", "EQIX_Equinix_ret_5d", "TM_Telephone_ret_1d", "EFFR_ret_1d", "AORD_AUS_zscore_60d", "SBUX_ret_5d", "HangSeng_HK_vol_20d", "VOD_Vodafone_zscore_60d", "ITT_ITTInc_ret_5d", "XLV_Health_zscore_60d", "NEE_NextEra_ret_20d", "EQR_Equity_ret_1d", "HD_ret_1d"], "is_new": true}, {"model_id": "new_h5_STRESS_RandomForest_N15_t2", "algo": "RandomForest", "regime": "STRESS", "horizon": 5, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "JNJ_ret_1d", "WTI_Oil_FRED_zscore_60d", "CPB_CampbellSoup_zscore_60d", "DE_Deere_ret_5d", "EWQ_France_zscore_60d", "Brent_Oil_FRED_ret_20d", "HD_ret_5d", "spx_vol_5d", "CI_Cigna_vol_20d", "GE_ret_1d", "EOG_EOGResources_ret_5d", "CCI_CrownCastle_vol_20d", "LMT_LockheedMartin_ret_1d"], "is_new": true}, {"model_id": "new_h5_STRESS_RandomForest_N15_t3", "algo": "RandomForest", "regime": "STRESS", "horizon": 5, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "JNJ_ret_1d", "Core_PCE_zscore_60d", "EWY_Korea_zscore_60d", "EWL_Switzerland_zscore_60d", "HangSeng_HK_vol_20d", "EWL_Switzerland_vol_20d", "TM_Telephone_vol_20d", "DIS_vol_20d", "LLY_zscore_60d", "BLK_BlackRock_zscore_60d", "BDX_Becton_Dickinson_ret_20d", "TED_Spread_vol_20d", "MSTR_Bitcoin3_ret_20d"], "is_new": true}, {"model_id": "new_h5_STRESS_RandomForest_N15_t4", "algo": "RandomForest", "regime": "STRESS", "horizon": 5, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "BTI_BritishAmerican_ret_5d", "BLK_BlackRock_zscore_60d", "EWA_Australia_zscore_60d", "EWY_Korea_zscore_60d", "Brent_Oil_FRED_ret_5d", "LOW_Lowes_ret_5d", "VRP_ma5", "HangSeng_HK_ret_5d", "XLY_Disc_vol_20d", "spx_momentum_3d", "DHR_ret_1d", "TXN_vol_20d", "EWA_Australia_ret_1d"], "is_new": true}, {"model_id": "new_h5_STRESS_RandomForest_N15_t5", "algo": "RandomForest", "regime": "STRESS", "horizon": 5, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "LMT_LockheedMartin_vol_20d", "DE_Deere_vol_20d", "PAYX_Paychex_ret_20d", "BLK_BlackRock_zscore_60d", "NVDA_vol_20d", "AMD_ret_1d", "PAYX_Paychex_vol_20d", "SBUX_ret_5d", "heston_var_ev_h7", "hmm_p_stress", "Brent_Oil_FRED_ret_5d", "GILD_Gilead_ret_20d", "WTI_Oil_FRED_zscore_60d"], "is_new": true}, {"model_id": "new_h5_STRESS_RandomForest_N15_t6", "algo": "RandomForest", "regime": "STRESS", "horizon": 5, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWM_Malaysia_zscore_60d", "MS_MorganStanley_zscore_60d", "XLK_Tech_zscore_60d", "CPB_CampbellSoup_zscore_60d", "SCHW_Schwab_ret_5d", "heston_var_ev_h5", "AXP_Amex_ret_20d", "EWQ_France_ret_20d", "DE_Deere_ret_5d", "IBEX_Spain_ret_20d", "EOG_EOGResources_vol_20d", "EQR_Equity_ret_1d", "EWM_Malaysia_vol_20d"], "is_new": true}, {"model_id": "new_h5_STRESS_RandomForest_N15_t7", "algo": "RandomForest", "regime": "STRESS", "horizon": 5, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "VOD_Vodafone_zscore_60d", "DOW_Price_zscore_60d", "CCI_CrownCastle_vol_20d", "CPB_CampbellSoup_vol_20d", "PFE_ret_1d", "BLK_BlackRock_zscore_60d", "ASX_Australia_vol_20d", "LUV_SouthwestAir_ret_5d", "EMR_Emerson_ret_20d", "HD_ret_5d", "INTC_ret_1d", "XOM_ret_1d", "MS_MorganStanley_ret_5d"], "is_new": true}, {"model_id": "new_h5_STRESS_RandomForest_N20_t0", "algo": "RandomForest", "regime": "STRESS", "horizon": 5, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "PPL_PPL_ret_1d", "Core_PCE_zscore_60d", "SBUX_ret_5d", "EOG_EOGResources_ret_5d", "ASX_Australia_vol_20d", "US3M_Rate_vol_20d", "TM_Telephone_ret_1d", "HD_ret_5d", "AMGN_Amgen_ret_1d", "EFFR_vol_20d", "PAYX_Paychex_ret_20d", "ASX_Australia_ret_5d", "heston_var_ev_h5", "US5Y_Rate_ret_5d", "GE_ret_1d", "EWC_Canada_zscore_60d", "CMCSA_ret_1d", "LMT_LockheedMartin_ret_1d"], "is_new": true}, {"model_id": "new_h5_STRESS_RandomForest_N20_t1", "algo": "RandomForest", "regime": "STRESS", "horizon": 5, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "DAX_Germany_vol_20d", "PPL_PPL_ret_1d", "T10Y2Y_Spread_ret_5d", "NOC_Northrop_ret_20d", "EQR_Equity_ret_1d", "LLY_zscore_60d", "spx_abs_ret_max_5d", "M_Macys_vol_20d", "CMCSA_ret_1d", "MO_AltriaMG_ret_1d", "NEE_NextEra_ret_20d", "EWQ_France_zscore_60d", "NWL_Newell_ret_20d", "US1Y_Rate_ret_5d", "TM_Telephone_vol_20d", "GE_ret_1d", "vix_mean_abs_ret_5d", "MSTR_Bitcoin3_ret_5d"], "is_new": true}, {"model_id": "new_h5_STRESS_RandomForest_N20_t2", "algo": "RandomForest", "regime": "STRESS", "horizon": 5, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "ITT_ITTInc_ret_5d", "MSTR_Bitcoin3_ret_1d", "US5Y_Rate_ret_5d", "MS_MorganStanley_ret_1d", "3M_vol_20d", "NOC_Northrop_ret_20d", "EFFR_vol_20d", "SPY_zscore_60d", "LMT_LockheedMartin_ret_1d", "EWM_Malaysia_vol_20d", "MSTR_Bitcoin3_ret_5d", "gjr_condvar_h1", "spx_momentum_3d", "SBUX_vol_20d", "IBEX_Spain_ret_20d", "XLB_Materials_zscore_60d", "EOG_EOGResources_ret_5d", "US30Y_Rate_ret_20d"], "is_new": true}, {"model_id": "new_h5_STRESS_RandomForest_N20_t3", "algo": "RandomForest", "regime": "STRESS", "horizon": 5, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "Brent_Oil_FRED_ret_20d", "EWG_Germany_ret_20d", "NFCI_ret_5d", "EWM_Malaysia_zscore_60d", "LMT_LockheedMartin_ret_1d", "HD_ret_5d", "EWA_Australia_ret_1d", "Core_PCE_zscore_60d", "EWS_Singapore_ret_5d", "CMCSA_ret_1d", "DHR_vol_20d", "SBUX_zscore_60d", "BA_ret_1d", "ORCL_vol_20d", "INTC_ret_5d", "IBEX_Spain_ret_20d", "Retail_Sales_zscore_60d", "TED_Spread_zscore_60d"], "is_new": true}, {"model_id": "new_h5_STRESS_RandomForest_N20_t4", "algo": "RandomForest", "regime": "STRESS", "horizon": 5, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "XLF_Fin_vol_20d", "CLX_Clorox_vol_20d", "AMGN_Amgen_ret_1d", "EWQ_France_zscore_60d", "AVB_AvalonBay_zscore_60d", "EWH_HongKong_ret_5d", "EWM_Malaysia_ret_1d", "QQQ_vol_20d", "VOD_Vodafone_zscore_60d", "Core_CPI_zscore_60d", "Michigan_Sentiment_ret_20d", "Brent_Oil_FRED_ret_20d", "SO_SouthernCo_ret_5d", "IYR_US_REIT2_zscore_60d", "3M_ret_5d", "EMR_Emerson_ret_20d", "XLV_Health_zscore_60d", "PG_ret_20d"], "is_new": true}, {"model_id": "new_h5_STRESS_RandomForest_N20_t5", "algo": "RandomForest", "regime": "STRESS", "horizon": 5, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EXC_Exelon_ret_1d", "HD_ret_20d", "3M_vol_20d", "VVIX_ret_20d", "EWH_HongKong_ret_5d", "AORD_AUS_zscore_60d", "INTC_ret_5d", "SBUX_ret_5d", "ORCL_zscore_60d", "NWL_Newell_ret_20d", "hmm_p_stress", "CLX_Clorox_vol_20d", "HD_zscore_60d", "AMD_ret_5d", "MSTR_Bitcoin3_ret_5d", "US7Y_Rate_ret_20d", "EFFR_ret_1d", "MS_MorganStanley_ret_5d"], "is_new": true}, {"model_id": "new_h5_STRESS_RandomForest_N20_t6", "algo": "RandomForest", "regime": "STRESS", "horizon": 5, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EFFR_ret_1d", "US1Y_Rate_ret_5d", "NOC_Northrop_ret_20d", "EWL_Switzerland_vol_20d", "LLY_zscore_60d", "SBUX_zscore_60d", "HD_ret_1d", "ASX_Australia_vol_20d", "SJM_JM_Smucker_ret_5d", "spx_vol_5d", "Industrial_Production_zscore_60d", "CPB_CampbellSoup_zscore_60d", "PAYX_Paychex_ret_20d", "HangSeng_HK_vol_20d", "FedFunds_zscore_60d", "HangSeng_HK_ret_1d", "HD_zscore_60d", "Brent_Oil_FRED_ret_5d"], "is_new": true}, {"model_id": "new_h5_STRESS_RandomForest_N20_t7", "algo": "RandomForest", "regime": "STRESS", "horizon": 5, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWS_Singapore_ret_5d", "spx_vol_5d", "LMT_LockheedMartin_ret_1d", "JNJ_ret_1d", "DIS_vol_20d", "MS_MorganStanley_ret_5d", "EWM_Malaysia_vol_20d", "PFE_ret_1d", "EWG_Germany_vol_20d", "TM_Telephone_ret_1d", "BLK_BlackRock_zscore_60d", "MSTR_Bitcoin3_ret_5d", "CPB_CampbellSoup_ret_5d", "SCHW_Schwab_ret_5d", "Retail_Sales_zscore_60d", "XLY_Disc_vol_20d", "WTI_Oil_FRED_zscore_60d", "SO_SouthernCo_ret_5d"], "is_new": true}, {"model_id": "new_h5_STRESS_RandomForest_N25_t0", "algo": "RandomForest", "regime": "STRESS", "horizon": 5, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "TGT_Target_zscore_60d", "CLX_Clorox_vol_20d", "BA_ret_1d", "AMD_ret_1d", "MS_MorganStanley_zscore_60d", "EWL_Switzerland_vol_20d", "CPB_CampbellSoup_zscore_60d", "3M_ret_5d", "INTC_ret_1d", "VVIX_ret_20d", "gjr_condvar_h1", "LMT_LockheedMartin_vol_20d", "AORD_AUS_zscore_60d", "PG_ret_20d", "Michigan_Sentiment_ret_20d", "DAX_Germany_vol_20d", "EXC_Exelon_ret_1d", "3M_vol_20d", "FedFunds_zscore_60d", "SO_SouthernCo_ret_5d", "Nikkei_Japan_vol_20d", "SLB_Schlumberger_ret_1d", "PLD_Prologis_ret_5d"], "is_new": true}, {"model_id": "new_h5_STRESS_RandomForest_N25_t1", "algo": "RandomForest", "regime": "STRESS", "horizon": 5, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWQ_France_zscore_60d", "EWM_Malaysia_ret_1d", "US1Y_Rate_ret_5d", "DIS_vol_20d", "CPB_CampbellSoup_zscore_60d", "BDX_Becton_Dickinson_ret_20d", "WTI_Oil_FRED_zscore_60d", "EQR_Equity_ret_1d", "BA_ret_1d", "AXP_Amex_ret_20d", "XOM_ret_1d", "spx_abs_ret_max_5d", "Core_CPI_zscore_60d", "HD_ret_1d", "XLB_Materials_zscore_60d", "TED_Spread_vol_20d", "XLV_Health_zscore_60d", "DHR_ret_1d", "gjr_condvar_h1", "MSTR_Bitcoin3_ret_20d", "spx_vol_5d", "LMT_LockheedMartin_ret_1d", "AMT_AmericanTower_ret_1d"], "is_new": true}, {"model_id": "new_h5_STRESS_RandomForest_N25_t2", "algo": "RandomForest", "regime": "STRESS", "horizon": 5, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "XLV_Health_zscore_60d", "T10Y2Y_Spread_ret_5d", "heston_var_ev_h5", "EWL_Switzerland_zscore_60d", "MSTR_Bitcoin3_ret_5d", "EXC_Exelon_ret_1d", "MS_MorganStanley_ret_5d", "XLB_Materials_zscore_60d", "AMD_ret_1d", "XOM_ret_20d", "NVDA_vol_20d", "CTAS_Cintas_vol_20d", "AMD_ret_5d", "XLK_Tech_zscore_60d", "Nikkei_Japan_vol_20d", "ASX_Australia_vol_20d", "spx_momentum_3d", "EWM_Malaysia_zscore_60d", "vix_acceleration_1d", "EFFR_ret_1d", "DIS_vol_20d", "EWH_HongKong_ret_5d", "US1Y_Rate_ret_20d"], "is_new": true}, {"model_id": "new_h5_STRESS_RandomForest_N25_t3", "algo": "RandomForest", "regime": "STRESS", "horizon": 5, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "MRK_Merck_zscore_60d", "EWJ_Japan_vol_20d", "PFE_ret_1d", "INTC_ret_5d", "IWM_SmallCap_vol_20d", "SJM_JM_Smucker_ret_1d", "DE_Deere_vol_20d", "PLD_Prologis_ret_5d", "ASX_Australia_vol_20d", "IYR_US_REIT2_zscore_60d", "XLK_Tech_zscore_60d", "DE_Deere_ret_5d", "spx_momentum_3d", "AORD_AUS_zscore_60d", "BTI_BritishAmerican_ret_20d", "TXN_vol_20d", "TM_Telephone_ret_1d", "PAYX_Paychex_ret_20d", "DOW_Price_zscore_60d", "AXP_Amex_ret_20d", "NEE_NextEra_ret_20d", "M_Macys_vol_20d", "LOW_Lowes_ret_5d"], "is_new": true}, {"model_id": "new_h5_STRESS_RandomForest_N25_t4", "algo": "RandomForest", "regime": "STRESS", "horizon": 5, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "PLD_Prologis_ret_5d", "TXN_vol_20d", "SLB_Schlumberger_ret_1d", "PPL_PPL_ret_1d", "DAX_Germany_vol_20d", "EWA_Australia_ret_1d", "AMD_ret_1d", "SO_SouthernCo_ret_5d", "XOM_ret_20d", "EXC_Exelon_ret_1d", "SJM_JM_Smucker_ret_5d", "EWH_HongKong_ret_5d", "XLB_Materials_zscore_60d", "SJM_JM_Smucker_ret_1d", "DE_Deere_ret_5d", "3M_ret_5d", "AVB_AvalonBay_zscore_60d", "MS_MorganStanley_ret_5d", "NWL_Newell_ret_20d", "EWJ_Japan_vol_20d", "XLV_Health_zscore_60d", "IYR_US_REIT2_zscore_60d", "Nikkei_Japan_vol_20d"], "is_new": true}, {"model_id": "new_h5_STRESS_RandomForest_N25_t5", "algo": "RandomForest", "regime": "STRESS", "horizon": 5, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "IBEX_Spain_ret_20d", "SJM_JM_Smucker_ret_5d", "VOD_Vodafone_zscore_60d", "EWS_Singapore_ret_5d", "M_Macys_vol_20d", "PLD_Prologis_ret_5d", "heston_var_ev_h3", "AMGN_Amgen_ret_1d", "Nikkei_Japan_zscore_60d", "BTI_BritishAmerican_ret_20d", "US3M_Rate_zscore_60d", "SCHW_Schwab_ret_5d", "LOW_Lowes_ret_20d", "AMD_ret_1d", "3M_ret_5d", "EWL_Switzerland_vol_20d", "TED_Spread_vol_20d", "Brent_Oil_FRED_ret_5d", "HangSeng_HK_ret_1d", "BDX_Becton_Dickinson_ret_20d", "EXC_Exelon_ret_1d", "EWG_Germany_vol_20d", "US1Y_Rate_ret_5d"], "is_new": true}, {"model_id": "new_h5_STRESS_RandomForest_N25_t6", "algo": "RandomForest", "regime": "STRESS", "horizon": 5, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWG_Germany_vol_20d", "BTI_BritishAmerican_ret_5d", "LMT_LockheedMartin_vol_20d", "3M_vol_20d", "AMD_ret_1d", "LOW_Lowes_ret_5d", "TGT_Target_zscore_60d", "EWH_HongKong_ret_5d", "MO_AltriaMG_ret_1d", "EWA_Australia_ret_1d", "HangSeng_HK_ret_5d", "TM_Telephone_vol_20d", "ITT_ITTInc_ret_5d", "US1Y_Rate_ret_5d", "HUM_Humana_ret_5d", "ES_Evergy_ret_1d", "IYM_BasicMaterials_ret_20d", "XLF_Fin_vol_20d", "MS_MorganStanley_zscore_60d", "vix_acceleration_1d", "GILD_Gilead_ret_20d", "BTI_BritishAmerican_ret_20d", "PAYX_Paychex_ret_20d"], "is_new": true}, {"model_id": "new_h5_STRESS_RandomForest_N25_t7", "algo": "RandomForest", "regime": "STRESS", "horizon": 5, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "MS_MorganStanley_ret_1d", "AORD_AUS_zscore_60d", "CPB_CampbellSoup_ret_5d", "T10Y2Y_Spread_ret_5d", "EWC_Canada_zscore_60d", "ES_Evergy_ret_1d", "LOW_Lowes_ret_20d", "Retail_Sales_zscore_60d", "US3Y_Rate_ret_5d", "SLB_Schlumberger_ret_5d", "Core_PCE_zscore_60d", "MSTR_Bitcoin3_ret_5d", "VVIX_ret_20d", "LMT_LockheedMartin_vol_20d", "EWM_Malaysia_zscore_60d", "LUV_SouthwestAir_ret_5d", "DOW_Price_zscore_60d", "EXC_Exelon_zscore_60d", "NEE_NextEra_ret_20d", "VRP_ma5", "EWA_Australia_ret_1d", "Nikkei_Japan_vol_20d", "BA_ret_1d"], "is_new": true}, {"model_id": "new_h5_STRESS_RandomForest_N30_t0", "algo": "RandomForest", "regime": "STRESS", "horizon": 5, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EXC_Exelon_zscore_60d", "3M_vol_20d", "BTI_BritishAmerican_ret_5d", "XLV_Health_zscore_60d", "EWQ_France_zscore_60d", "US3Y_Rate_ret_5d", "INTC_ret_1d", "US30Y_Rate_ret_20d", "US5Y_Rate_ret_5d", "SLB_Schlumberger_ret_5d", "CCI_CrownCastle_vol_20d", "heston_ev_h3", "HD_zscore_60d", "EWQ_France_ret_20d", "ES_Evergy_ret_1d", "HD_ret_1d", "HangSeng_HK_vol_20d", "LOW_Lowes_ret_5d", "SJM_JM_Smucker_ret_1d", "IBEX_Spain_ret_20d", "EWM_Malaysia_zscore_60d", "VVIX_ret_20d", "EQR_Equity_ret_1d", "heston_var_ev_h7", "LMT_LockheedMartin_vol_20d", "Brent_Oil_FRED_ret_5d", "EFFR_vol_20d", "EFFR_ret_1d"], "is_new": true}, {"model_id": "new_h5_STRESS_RandomForest_N30_t1", "algo": "RandomForest", "regime": "STRESS", "horizon": 5, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "US7Y_Rate_ret_20d", "EWL_Switzerland_zscore_60d", "gjr_condvar_h1", "SJM_JM_Smucker_ret_5d", "EMR_Emerson_ret_20d", "CPB_CampbellSoup_vol_20d", "AXP_Amex_vol_20d", "ORCL_vol_20d", "EFFR_ret_1d", "TM_Telephone_ret_1d", "MSTR_Bitcoin3_ret_20d", "US1Y_Rate_ret_20d", "LLY_zscore_60d", "DE_Deere_vol_20d", "CPB_CampbellSoup_ret_5d", "AVB_AvalonBay_zscore_60d", "XLK_Tech_zscore_60d", "PFE_ret_1d", "EWM_Malaysia_ret_1d", "Industrial_Production_zscore_60d", "ASX_Australia_ret_5d", "Brent_Oil_FRED_ret_5d", "EWG_Germany_ret_20d", "EQR_Equity_ret_1d", "LMT_LockheedMartin_vol_20d", "PG_ret_20d", "AMT_AmericanTower_ret_1d", "HangSeng_HK_ret_5d"], "is_new": true}, {"model_id": "new_h5_STRESS_RandomForest_N30_t2", "algo": "RandomForest", "regime": "STRESS", "horizon": 5, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "DOW_Price_zscore_60d", "HD_ret_1d", "EWH_HongKong_ret_5d", "AMGN_Amgen_ret_1d", "EQR_Equity_ret_1d", "Industrial_Production_zscore_60d", "XLV_Health_zscore_60d", "IWM_SmallCap_vol_20d", "SLB_Schlumberger_ret_1d", "T10Y2Y_Spread_ret_5d", "US1Y_Rate_ret_5d", "LUV_SouthwestAir_ret_5d", "heston_var_ev_h5", "PAYX_Paychex_ret_20d", "spx_vol_5d", "VVIX_ret_20d", "EWL_Switzerland_zscore_60d", "SBUX_zscore_60d", "GD_GeneralDynamics_zscore_60d", "EOG_EOGResources_ret_5d", "M_Macys_vol_20d", "3M_vol_20d", "TXN_vol_20d", "LOW_Lowes_ret_20d", "LMT_LockheedMartin_vol_20d", "AXP_Amex_ret_20d", "DE_Deere_vol_20d", "MS_MorganStanley_ret_5d"], "is_new": true}, {"model_id": "new_h5_STRESS_RandomForest_N30_t3", "algo": "RandomForest", "regime": "STRESS", "horizon": 5, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWQ_France_ret_20d", "INTC_ret_5d", "EWL_Switzerland_vol_20d", "US3Y_Rate_ret_5d", "LOW_Lowes_ret_5d", "VVIX_ret_20d", "vix_acceleration_1d", "NEE_NextEra_ret_20d", "XLB_Materials_zscore_60d", "DAX_Germany_vol_20d", "PLD_Prologis_ret_5d", "EMR_Emerson_ret_20d", "AMD_ret_5d", "PPL_PPL_ret_1d", "BLK_BlackRock_zscore_60d", "EWY_Korea_ret_20d", "ORCL_vol_20d", "SBUX_vol_20d", "US1Y_Rate_ret_5d", "SJM_JM_Smucker_ret_5d", "EWG_Germany_vol_20d", "QQQ_vol_20d", "EWA_Australia_zscore_60d", "MS_MorganStanley_ret_1d", "EWL_Switzerland_zscore_60d", "AXP_Amex_vol_20d", "CPB_CampbellSoup_ret_20d", "LLY_zscore_60d"], "is_new": true}, {"model_id": "new_h5_STRESS_RandomForest_N30_t4", "algo": "RandomForest", "regime": "STRESS", "horizon": 5, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWG_Germany_ret_20d", "PLD_Prologis_ret_5d", "HD_ret_20d", "TM_Telephone_vol_20d", "CMCSA_ret_1d", "DAX_Germany_vol_20d", "TGT_Target_zscore_60d", "HangSeng_HK_ret_5d", "GILD_Gilead_ret_20d", "TXN_vol_20d", "SCHW_Schwab_ret_5d", "heston_var_ev_h7", "US6M_Rate_ret_20d", "AMGN_Amgen_ret_1d", "LOW_Lowes_ret_20d", "DOW_Price_zscore_60d", "US5Y_Rate_ret_5d", "CPB_CampbellSoup_ret_5d", "spx_momentum_3d", "spx_vol_5d", "HD_zscore_60d", "AXP_Amex_vol_20d", "US3M_Rate_zscore_60d", "AMT_AmericanTower_ret_1d", "BLK_BlackRock_zscore_60d", "ENB_EnbridgeInc_ret_1d", "JNJ_ret_1d", "ITT_ITTInc_ret_5d"], "is_new": true}, {"model_id": "new_h5_STRESS_RandomForest_N30_t5", "algo": "RandomForest", "regime": "STRESS", "horizon": 5, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "PG_ret_20d", "EWC_Canada_zscore_60d", "CTAS_Cintas_vol_20d", "EWJ_Japan_vol_20d", "AMT_AmericanTower_ret_1d", "EQIX_Equinix_ret_5d", "EFFR_vol_20d", "AORD_AUS_zscore_60d", "EWM_Malaysia_zscore_60d", "PAYX_Paychex_zscore_60d", "EWS_Singapore_ret_5d", "LLY_zscore_60d", "HangSeng_HK_ret_5d", "AVB_AvalonBay_zscore_60d", "LUV_SouthwestAir_ret_5d", "heston_var_ev_h3", "EWG_Germany_ret_20d", "EWY_Korea_zscore_60d", "3M_ret_5d", "CPB_CampbellSoup_ret_20d", "NOC_Northrop_ret_20d", "HD_ret_1d", "PFE_ret_1d", "TXN_vol_20d", "LOW_Lowes_ret_5d", "BLK_BlackRock_zscore_60d", "VRP_ma5", "ASX_Australia_ret_5d"], "is_new": true}, {"model_id": "new_h5_STRESS_RandomForest_N30_t6", "algo": "RandomForest", "regime": "STRESS", "horizon": 5, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "BTI_BritishAmerican_ret_5d", "XLY_Disc_vol_20d", "TM_Telephone_ret_1d", "MSTR_Bitcoin3_ret_20d", "LMT_LockheedMartin_vol_20d", "SLB_Schlumberger_ret_1d", "INTC_ret_5d", "Brent_Oil_FRED_ret_20d", "AMT_AmericanTower_ret_1d", "DE_Deere_ret_5d", "vix_mean_abs_ret_5d", "HD_ret_5d", "M_Macys_vol_20d", "ITT_ITTInc_ret_5d", "EXC_Exelon_zscore_60d", "spx_vol_5d", "CPB_CampbellSoup_vol_20d", "US3M_Rate_vol_20d", "EWG_Germany_vol_20d", "EOG_EOGResources_vol_20d", "ENB_EnbridgeInc_ret_1d", "heston_var_ev_h3", "MS_MorganStanley_ret_5d", "gjr_condvar_h1", "spx_abs_ret_max_5d", "EOG_EOGResources_ret_5d", "heston_var_ev_h7", "AORD_AUS_zscore_60d"], "is_new": true}, {"model_id": "new_h5_STRESS_RandomForest_N30_t7", "algo": "RandomForest", "regime": "STRESS", "horizon": 5, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "XLB_Materials_zscore_60d", "SLB_Schlumberger_ret_5d", "PAYX_Paychex_ret_20d", "US3M_Rate_vol_20d", "INTC_ret_1d", "AMT_AmericanTower_ret_1d", "HangSeng_HK_ret_1d", "PLD_Prologis_ret_5d", "heston_var_ev_h3", "ASX_Australia_ret_5d", "AMD_ret_1d", "M_Macys_vol_20d", "EQR_Equity_ret_1d", "DHR_ret_1d", "ITT_ITTInc_ret_5d", "AXP_Amex_ret_20d", "Brent_Oil_FRED_ret_5d", "PAYX_Paychex_zscore_60d", "SCHW_Schwab_ret_5d", "IYR_US_REIT2_zscore_60d", "XLF_Fin_vol_20d", "heston_var_ev_h5", "HD_ret_20d", "TXN_vol_20d", "ASX_Australia_vol_20d", "EQIX_Equinix_ret_5d", "EWC_Canada_zscore_60d", "spx_momentum_3d"], "is_new": true}, {"model_id": "new_h5_STRESS_LogisticRegression_N5_t0", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 5, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "heston_var_ev_h5", "HD_ret_1d", "SBUX_vol_20d"], "is_new": true}, {"model_id": "new_h5_STRESS_LogisticRegression_N5_t1", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 5, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "Nikkei_Japan_zscore_60d", "BA_ret_1d", "LOW_Lowes_ret_20d"], "is_new": true}, {"model_id": "new_h5_STRESS_LogisticRegression_N5_t2", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 5, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "PPL_PPL_ret_1d", "EWG_Germany_vol_20d", "US5Y_Rate_ret_5d"], "is_new": true}, {"model_id": "new_h5_STRESS_LogisticRegression_N5_t3", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 5, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EFFR_vol_20d", "IWM_SmallCap_vol_20d", "VRP_ma5"], "is_new": true}, {"model_id": "new_h5_STRESS_LogisticRegression_N5_t4", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 5, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "SO_SouthernCo_ret_5d", "WTI_Oil_FRED_zscore_60d", "TM_Telephone_ret_1d"], "is_new": true}, {"model_id": "new_h5_STRESS_LogisticRegression_N5_t5", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 5, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EFFR_ret_1d", "HD_zscore_60d", "EXC_Exelon_ret_1d"], "is_new": true}, {"model_id": "new_h5_STRESS_LogisticRegression_N5_t6", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 5, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "XLF_Fin_vol_20d", "NFCI_ret_5d", "CI_Cigna_vol_20d"], "is_new": true}, {"model_id": "new_h5_STRESS_LogisticRegression_N5_t7", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 5, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EXC_Exelon_zscore_60d", "AORD_AUS_zscore_60d", "TM_Telephone_vol_20d"], "is_new": true}, {"model_id": "new_h5_STRESS_LogisticRegression_N8_t0", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 5, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWM_Malaysia_ret_1d", "DAX_Germany_vol_20d", "INTC_ret_5d", "AXP_Amex_ret_20d", "DHR_ret_1d", "XOM_ret_20d"], "is_new": true}, {"model_id": "new_h5_STRESS_LogisticRegression_N8_t1", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 5, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "XLF_Fin_vol_20d", "Core_CPI_zscore_60d", "US3M_Rate_vol_20d", "CPB_CampbellSoup_ret_5d", "MSTR_Bitcoin3_ret_20d", "LMT_LockheedMartin_vol_20d"], "is_new": true}, {"model_id": "new_h5_STRESS_LogisticRegression_N8_t2", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 5, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "NFCI_ret_5d", "DE_Deere_vol_20d", "BDX_Becton_Dickinson_ret_20d", "US5Y_Rate_ret_5d", "vix_mean_abs_ret_5d", "INTC_ret_1d"], "is_new": true}, {"model_id": "new_h5_STRESS_LogisticRegression_N8_t3", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 5, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "BLK_BlackRock_zscore_60d", "PAYX_Paychex_vol_20d", "EFFR_vol_20d", "LMT_LockheedMartin_vol_20d", "VVIX_ret_20d", "EWA_Australia_ret_1d"], "is_new": true}, {"model_id": "new_h5_STRESS_LogisticRegression_N8_t4", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 5, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "AXP_Amex_vol_20d", "NOC_Northrop_ret_20d", "T_ret_1d", "ES_Evergy_ret_1d", "DHR_vol_20d", "AMD_ret_5d"], "is_new": true}, {"model_id": "new_h5_STRESS_LogisticRegression_N8_t5", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 5, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EMR_Emerson_ret_20d", "US1Y_Rate_ret_5d", "CTAS_Cintas_vol_20d", "EWG_Germany_vol_20d", "EWY_Korea_zscore_60d", "US3Y_Rate_ret_5d"], "is_new": true}, {"model_id": "new_h5_STRESS_LogisticRegression_N8_t6", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 5, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "DOW_Price_zscore_60d", "EWM_Malaysia_ret_1d", "DIS_vol_20d", "MSTR_Bitcoin3_ret_1d", "CI_Cigna_vol_20d", "ASX_Australia_ret_5d"], "is_new": true}, {"model_id": "new_h5_STRESS_LogisticRegression_N8_t7", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 5, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "TED_Spread_vol_20d", "NFCI_ret_5d", "EOG_EOGResources_vol_20d", "3M_vol_20d", "INTC_ret_5d", "AXP_Amex_ret_20d"], "is_new": true}, {"model_id": "new_h5_STRESS_LogisticRegression_N10_t0", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 5, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "BTI_BritishAmerican_ret_20d", "EWS_Singapore_ret_5d", "AXP_Amex_vol_20d", "US3M_Rate_vol_20d", "gjr_condvar_h1", "VOD_Vodafone_zscore_60d", "XLY_Disc_vol_20d", "MS_MorganStanley_ret_5d"], "is_new": true}, {"model_id": "new_h5_STRESS_LogisticRegression_N10_t1", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 5, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "HD_zscore_60d", "heston_var_ev_h5", "XLK_Tech_zscore_60d", "SJM_JM_Smucker_ret_1d", "SBUX_zscore_60d", "AMGN_Amgen_ret_1d", "QQQ_vol_20d", "AMT_AmericanTower_ret_1d"], "is_new": true}, {"model_id": "new_h5_STRESS_LogisticRegression_N10_t2", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 5, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EOG_EOGResources_vol_20d", "Brent_Oil_FRED_ret_20d", "LOW_Lowes_ret_20d", "IYM_BasicMaterials_ret_20d", "DE_Deere_vol_20d", "SBUX_vol_20d", "EXC_Exelon_ret_1d", "EFFR_ret_1d"], "is_new": true}, {"model_id": "new_h5_STRESS_LogisticRegression_N10_t3", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 5, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "SBUX_vol_20d", "LMT_LockheedMartin_vol_20d", "US3M_Rate_zscore_60d", "HangSeng_HK_ret_5d", "Core_CPI_zscore_60d", "spx_vol_5d", "AMD_ret_5d", "EWY_Korea_zscore_60d"], "is_new": true}, {"model_id": "new_h5_STRESS_LogisticRegression_N10_t4", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 5, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "ITT_ITTInc_ret_5d", "HangSeng_HK_ret_5d", "SJM_JM_Smucker_ret_1d", "TED_Spread_vol_20d", "DIS_vol_20d", "XLF_Fin_vol_20d", "HangSeng_HK_ret_1d", "BDX_Becton_Dickinson_ret_20d"], "is_new": true}, {"model_id": "new_h5_STRESS_LogisticRegression_N10_t5", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 5, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "Brent_Oil_FRED_ret_20d", "spx_momentum_3d", "CPB_CampbellSoup_zscore_60d", "TED_Spread_zscore_60d", "PAYX_Paychex_ret_20d", "BDX_Becton_Dickinson_ret_20d", "AMD_ret_5d", "ORCL_zscore_60d"], "is_new": true}, {"model_id": "new_h5_STRESS_LogisticRegression_N10_t6", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 5, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "CMCSA_ret_1d", "T_ret_1d", "AVB_AvalonBay_zscore_60d", "EFFR_ret_1d", "ASX_Australia_ret_5d", "EWA_Australia_ret_1d", "Core_CPI_zscore_60d", "MRK_Merck_zscore_60d"], "is_new": true}, {"model_id": "new_h5_STRESS_LogisticRegression_N10_t7", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 5, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "INTC_ret_5d", "ES_Evergy_ret_1d", "Core_CPI_zscore_60d", "NFCI_ret_5d", "EWC_Canada_zscore_60d", "MO_AltriaMG_ret_1d", "XLY_Disc_vol_20d", "EFFR_vol_20d"], "is_new": true}, {"model_id": "new_h5_STRESS_LogisticRegression_N12_t0", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 5, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "LMT_LockheedMartin_vol_20d", "AORD_AUS_zscore_60d", "PAYX_Paychex_zscore_60d", "ORCL_vol_20d", "ITT_ITTInc_ret_5d", "HangSeng_HK_vol_20d", "PCAR_PaccarInc_ret_5d", "TED_Spread_zscore_60d", "SO_SouthernCo_ret_5d", "US5Y_Rate_ret_5d"], "is_new": true}, {"model_id": "new_h5_STRESS_LogisticRegression_N12_t1", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 5, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "XOM_ret_1d", "TM_Telephone_ret_1d", "US7Y_Rate_ret_20d", "PLD_Prologis_ret_5d", "SBUX_zscore_60d", "LOW_Lowes_ret_5d", "vix_acceleration_1d", "Core_PCE_zscore_60d", "HangSeng_HK_vol_20d", "DE_Deere_vol_20d"], "is_new": true}, {"model_id": "new_h5_STRESS_LogisticRegression_N12_t2", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 5, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "DIS_vol_20d", "EFFR_vol_20d", "Core_CPI_zscore_60d", "EMR_Emerson_ret_20d", "3M_ret_5d", "EWA_Australia_zscore_60d", "PPL_PPL_ret_1d", "EWJ_Japan_vol_20d", "NFCI_ret_5d", "AMD_ret_5d"], "is_new": true}, {"model_id": "new_h5_STRESS_LogisticRegression_N12_t3", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 5, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "ES_Evergy_ret_1d", "EWQ_France_zscore_60d", "EQIX_Equinix_ret_5d", "EWS_Singapore_ret_5d", "DE_Deere_ret_5d", "XOM_ret_1d", "CPB_CampbellSoup_zscore_60d", "HD_ret_5d", "DE_Deere_vol_20d", "heston_var_ev_h3"], "is_new": true}, {"model_id": "new_h5_STRESS_LogisticRegression_N12_t4", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 5, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "VOD_Vodafone_zscore_60d", "EWJ_Japan_vol_20d", "US3Y_Rate_ret_5d", "IWM_SmallCap_vol_20d", "EWL_Switzerland_zscore_60d", "SLB_Schlumberger_ret_5d", "HD_zscore_60d", "EWA_Australia_zscore_60d", "AMD_ret_5d", "SBUX_ret_5d"], "is_new": true}, {"model_id": "new_h5_STRESS_LogisticRegression_N12_t5", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 5, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "AMD_ret_1d", "XLF_Fin_vol_20d", "AMZN_ret_5d", "EWS_Singapore_ret_5d", "IYM_BasicMaterials_ret_20d", "CTAS_Cintas_vol_20d", "hmm_p_stress", "GD_GeneralDynamics_zscore_60d", "VVIX_ret_20d", "BA_ret_1d"], "is_new": true}, {"model_id": "new_h5_STRESS_LogisticRegression_N12_t6", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 5, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "CPB_CampbellSoup_ret_20d", "EWY_Korea_ret_20d", "US3M_Rate_zscore_60d", "EWA_Australia_zscore_60d", "MSTR_Bitcoin3_ret_20d", "ASX_Australia_vol_20d", "LOW_Lowes_ret_20d", "QQQ_vol_20d", "DIS_vol_20d", "BA_ret_1d"], "is_new": true}, {"model_id": "new_h5_STRESS_LogisticRegression_N12_t7", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 5, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "Brent_Oil_FRED_ret_20d", "LOW_Lowes_ret_5d", "NFCI_ret_5d", "SBUX_zscore_60d", "EOG_EOGResources_vol_20d", "EWQ_France_ret_20d", "SJM_JM_Smucker_ret_5d", "Nikkei_Japan_vol_20d", "LOW_Lowes_ret_20d", "TGT_Target_zscore_60d"], "is_new": true}, {"model_id": "new_h5_STRESS_LogisticRegression_N15_t0", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 5, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "CPB_CampbellSoup_ret_20d", "MSTR_Bitcoin3_ret_1d", "Brent_Oil_FRED_ret_5d", "SJM_JM_Smucker_ret_5d", "BDX_Becton_Dickinson_ret_20d", "EWA_Australia_ret_1d", "ASX_Australia_ret_5d", "vix_mean_abs_ret_5d", "EWL_Switzerland_vol_20d", "TED_Spread_vol_20d", "ORCL_zscore_60d", "US6M_Rate_ret_20d", "GILD_Gilead_ret_20d"], "is_new": true}, {"model_id": "new_h5_STRESS_LogisticRegression_N15_t1", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 5, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "3M_vol_20d", "LMT_LockheedMartin_vol_20d", "HUM_Humana_ret_5d", "SCHW_Schwab_ret_5d", "LOW_Lowes_ret_20d", "MRK_Merck_zscore_60d", "Brent_Oil_FRED_ret_20d", "IYR_US_REIT2_zscore_60d", "spx_vol_5d", "M_Macys_vol_20d", "XOM_ret_1d", "T10Y2Y_Spread_ret_5d", "EWM_Malaysia_ret_1d"], "is_new": true}, {"model_id": "new_h5_STRESS_LogisticRegression_N15_t2", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 5, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EMR_Emerson_ret_20d", "SLB_Schlumberger_ret_1d", "XOM_ret_20d", "3M_vol_20d", "XLF_Fin_vol_20d", "BTI_BritishAmerican_ret_5d", "US3M_Rate_vol_20d", "CI_Cigna_vol_20d", "PLD_Prologis_ret_5d", "PPL_PPL_ret_1d", "AORD_AUS_zscore_60d", "CCI_CrownCastle_vol_20d", "US5Y_Rate_ret_5d"], "is_new": true}, {"model_id": "new_h5_STRESS_LogisticRegression_N15_t3", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 5, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "heston_var_ev_h7", "Core_CPI_zscore_60d", "PCAR_PaccarInc_ret_5d", "VRP_ma5", "Industrial_Production_zscore_60d", "Michigan_Sentiment_ret_20d", "SJM_JM_Smucker_ret_1d", "SPY_zscore_60d", "IYM_BasicMaterials_ret_20d", "heston_ev_h3", "US30Y_Rate_ret_20d", "ORCL_zscore_60d", "MS_MorganStanley_zscore_60d"], "is_new": true}, {"model_id": "new_h5_STRESS_LogisticRegression_N15_t4", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 5, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "PAYX_Paychex_zscore_60d", "SBUX_zscore_60d", "HD_ret_20d", "EXC_Exelon_zscore_60d", "IBEX_Spain_ret_20d", "CPB_CampbellSoup_zscore_60d", "LLY_zscore_60d", "MSTR_Bitcoin3_ret_1d", "LOW_Lowes_ret_5d", "HD_ret_5d", "EWJ_Japan_vol_20d", "SBUX_ret_5d", "IYR_US_REIT2_zscore_60d"], "is_new": true}, {"model_id": "new_h5_STRESS_LogisticRegression_N15_t5", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 5, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWL_Switzerland_vol_20d", "DIS_vol_20d", "TM_Telephone_ret_1d", "Core_CPI_zscore_60d", "SLB_Schlumberger_ret_1d", "AMD_ret_5d", "CPB_CampbellSoup_ret_20d", "EQR_Equity_ret_1d", "XLV_Health_zscore_60d", "LUV_SouthwestAir_ret_5d", "LMT_LockheedMartin_vol_20d", "SJM_JM_Smucker_ret_5d", "GE_ret_1d"], "is_new": true}, {"model_id": "new_h5_STRESS_LogisticRegression_N15_t6", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 5, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWA_Australia_ret_1d", "IWM_SmallCap_vol_20d", "ENB_EnbridgeInc_ret_1d", "MS_MorganStanley_ret_1d", "FedFunds_zscore_60d", "ASX_Australia_ret_5d", "PAYX_Paychex_vol_20d", "CPB_CampbellSoup_zscore_60d", "HangSeng_HK_ret_5d", "XOM_ret_20d", "US3Y_Rate_ret_5d", "CI_Cigna_vol_20d", "EWC_Canada_zscore_60d"], "is_new": true}, {"model_id": "new_h5_STRESS_LogisticRegression_N15_t7", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 5, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "MRK_Merck_zscore_60d", "DHR_vol_20d", "SJM_JM_Smucker_ret_1d", "IBEX_Spain_ret_20d", "BDX_Becton_Dickinson_ret_20d", "FedFunds_zscore_60d", "LOW_Lowes_ret_20d", "Nikkei_Japan_zscore_60d", "EWY_Korea_zscore_60d", "ITT_ITTInc_ret_5d", "HD_ret_1d", "EWJ_Japan_vol_20d", "BTI_BritishAmerican_ret_5d"], "is_new": true}, {"model_id": "new_h5_STRESS_LogisticRegression_N20_t0", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 5, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "TED_Spread_zscore_60d", "EQIX_Equinix_ret_5d", "NWL_Newell_ret_20d", "INTC_ret_5d", "CPB_CampbellSoup_ret_5d", "EFFR_ret_1d", "VOD_Vodafone_zscore_60d", "EXC_Exelon_ret_1d", "PAYX_Paychex_zscore_60d", "ES_Evergy_ret_1d", "US3Y_Rate_ret_5d", "US7Y_Rate_ret_20d", "DE_Deere_ret_5d", "Michigan_Sentiment_ret_20d", "EWM_Malaysia_ret_1d", "EWA_Australia_zscore_60d", "ASX_Australia_ret_5d", "spx_abs_ret_max_5d"], "is_new": true}, {"model_id": "new_h5_STRESS_LogisticRegression_N20_t1", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 5, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWL_Switzerland_vol_20d", "XLF_Fin_vol_20d", "CI_Cigna_vol_20d", "ES_Evergy_ret_1d", "DIS_vol_20d", "EFFR_ret_1d", "ORCL_zscore_60d", "Nikkei_Japan_zscore_60d", "CPB_CampbellSoup_ret_5d", "LMT_LockheedMartin_ret_1d", "US1Y_Rate_ret_20d", "EXC_Exelon_ret_1d", "DOW_Price_zscore_60d", "LLY_zscore_60d", "CLX_Clorox_vol_20d", "TM_Telephone_vol_20d", "QQQ_vol_20d", "HUM_Humana_ret_5d"], "is_new": true}, {"model_id": "new_h5_STRESS_LogisticRegression_N20_t2", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 5, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "US30Y_Rate_ret_20d", "IYM_BasicMaterials_ret_20d", "SLB_Schlumberger_ret_1d", "EWL_Switzerland_vol_20d", "CLX_Clorox_vol_20d", "EOG_EOGResources_ret_5d", "XOM_ret_1d", "SBUX_ret_5d", "LMT_LockheedMartin_ret_1d", "SBUX_zscore_60d", "AMZN_ret_5d", "EWG_Germany_ret_20d", "HD_zscore_60d", "AORD_AUS_zscore_60d", "PAYX_Paychex_ret_20d", "Core_PCE_zscore_60d", "TM_Telephone_ret_1d", "CMCSA_ret_1d"], "is_new": true}, {"model_id": "new_h5_STRESS_LogisticRegression_N20_t3", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 5, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "M_Macys_vol_20d", "DIS_vol_20d", "heston_var_ev_h3", "ORCL_zscore_60d", "spx_abs_ret_max_5d", "GD_GeneralDynamics_zscore_60d", "T10Y2Y_Spread_ret_5d", "CPB_CampbellSoup_zscore_60d", "US30Y_Rate_ret_20d", "PG_ret_20d", "EWA_Australia_ret_1d", "Core_CPI_zscore_60d", "Retail_Sales_zscore_60d", "US1Y_Rate_ret_20d", "INTC_ret_1d", "DE_Deere_vol_20d", "VRP_ma5", "Nikkei_Japan_zscore_60d"], "is_new": true}, {"model_id": "new_h5_STRESS_LogisticRegression_N20_t4", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 5, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "T_ret_1d", "INTC_ret_5d", "DAX_Germany_vol_20d", "VRP_ma5", "EFFR_ret_1d", "Michigan_Sentiment_ret_20d", "BTI_BritishAmerican_ret_20d", "SBUX_vol_20d", "vix_acceleration_1d", "LMT_LockheedMartin_vol_20d", "TM_Telephone_vol_20d", "M_Macys_vol_20d", "WTI_Oil_FRED_zscore_60d", "DOW_Price_zscore_60d", "PAYX_Paychex_zscore_60d", "HD_ret_20d", "EMR_Emerson_ret_20d", "spx_momentum_3d"], "is_new": true}, {"model_id": "new_h5_STRESS_LogisticRegression_N20_t5", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 5, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "XLY_Disc_vol_20d", "Core_PCE_zscore_60d", "MS_MorganStanley_ret_5d", "DAX_Germany_vol_20d", "AVB_AvalonBay_zscore_60d", "DHR_ret_1d", "SO_SouthernCo_ret_5d", "CCI_CrownCastle_vol_20d", "CPB_CampbellSoup_zscore_60d", "CPB_CampbellSoup_ret_5d", "SBUX_zscore_60d", "ASX_Australia_ret_5d", "HangSeng_HK_ret_1d", "AMT_AmericanTower_ret_1d", "HD_zscore_60d", "SJM_JM_Smucker_ret_1d", "TXN_vol_20d", "spx_abs_ret_max_5d"], "is_new": true}, {"model_id": "new_h5_STRESS_LogisticRegression_N20_t6", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 5, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "HangSeng_HK_ret_1d", "SLB_Schlumberger_ret_5d", "AXP_Amex_vol_20d", "MRK_Merck_zscore_60d", "DE_Deere_vol_20d", "Core_PCE_zscore_60d", "PG_ret_20d", "3M_ret_5d", "DAX_Germany_zscore_60d", "AXP_Amex_ret_20d", "SJM_JM_Smucker_ret_1d", "EWQ_France_zscore_60d", "US3M_Rate_vol_20d", "SBUX_ret_5d", "BDX_Becton_Dickinson_ret_20d", "XOM_ret_20d", "CCI_CrownCastle_vol_20d", "NOC_Northrop_ret_20d"], "is_new": true}, {"model_id": "new_h5_STRESS_LogisticRegression_N20_t7", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 5, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "DE_Deere_ret_5d", "US6M_Rate_ret_20d", "INTC_ret_1d", "SO_SouthernCo_ret_5d", "BTI_BritishAmerican_ret_20d", "BA_ret_1d", "PAYX_Paychex_zscore_60d", "Retail_Sales_zscore_60d", "CTAS_Cintas_vol_20d", "MS_MorganStanley_zscore_60d", "spx_momentum_3d", "SCHW_Schwab_ret_5d", "ORCL_vol_20d", "SBUX_zscore_60d", "EXC_Exelon_ret_1d", "Michigan_Sentiment_ret_20d", "MSTR_Bitcoin3_ret_5d", "GD_GeneralDynamics_zscore_60d"], "is_new": true}, {"model_id": "new_h5_STRESS_LogisticRegression_N25_t0", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 5, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "GD_GeneralDynamics_zscore_60d", "vix_acceleration_1d", "DE_Deere_vol_20d", "HD_ret_5d", "JNJ_ret_1d", "BDX_Becton_Dickinson_ret_20d", "GE_ret_1d", "XLV_Health_zscore_60d", "SLB_Schlumberger_ret_1d", "SLB_Schlumberger_ret_5d", "Michigan_Sentiment_ret_20d", "HD_zscore_60d", "LMT_LockheedMartin_vol_20d", "EWL_Switzerland_zscore_60d", "SJM_JM_Smucker_ret_5d", "NWL_Newell_ret_20d", "US30Y_Rate_ret_20d", "XOM_ret_20d", "INTC_ret_1d", "CPB_CampbellSoup_zscore_60d", "EOG_EOGResources_vol_20d", "NOC_Northrop_ret_20d", "IYM_BasicMaterials_ret_20d"], "is_new": true}, {"model_id": "new_h5_STRESS_LogisticRegression_N25_t1", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 5, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "PAYX_Paychex_ret_20d", "EWQ_France_ret_20d", "CPB_CampbellSoup_zscore_60d", "JNJ_ret_1d", "MO_AltriaMG_ret_1d", "EOG_EOGResources_vol_20d", "INTC_ret_1d", "GE_ret_1d", "CMCSA_ret_1d", "PLD_Prologis_ret_5d", "TM_Telephone_vol_20d", "EWJ_Japan_vol_20d", "EQR_Equity_ret_1d", "CLX_Clorox_vol_20d", "CCI_CrownCastle_vol_20d", "DOW_Price_zscore_60d", "MSTR_Bitcoin3_ret_20d", "TM_Telephone_ret_1d", "spx_vol_5d", "US1Y_Rate_ret_20d", "NFCI_ret_5d", "SPY_zscore_60d", "spx_abs_ret_max_5d"], "is_new": true}, {"model_id": "new_h5_STRESS_LogisticRegression_N25_t2", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 5, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "IWM_SmallCap_vol_20d", "TED_Spread_vol_20d", "JNJ_ret_1d", "NOC_Northrop_ret_20d", "VRP_ma5", "SBUX_zscore_60d", "SBUX_vol_20d", "BTI_BritishAmerican_ret_20d", "3M_ret_5d", "HD_ret_5d", "MO_AltriaMG_ret_1d", "MSTR_Bitcoin3_ret_1d", "SLB_Schlumberger_ret_1d", "US1Y_Rate_ret_20d", "EQR_Equity_ret_1d", "ORCL_vol_20d", "EWQ_France_ret_20d", "HangSeng_HK_ret_5d", "EQIX_Equinix_ret_5d", "XLB_Materials_zscore_60d", "XLV_Health_zscore_60d", "EWG_Germany_vol_20d", "LOW_Lowes_ret_5d"], "is_new": true}, {"model_id": "new_h5_STRESS_LogisticRegression_N25_t3", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 5, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "AMGN_Amgen_ret_1d", "IYM_BasicMaterials_ret_20d", "DE_Deere_vol_20d", "LMT_LockheedMartin_ret_1d", "CI_Cigna_vol_20d", "EWH_HongKong_ret_5d", "MS_MorganStanley_ret_5d", "EQR_Equity_ret_1d", "QQQ_vol_20d", "EQIX_Equinix_ret_5d", "IWM_SmallCap_vol_20d", "TGT_Target_zscore_60d", "Michigan_Sentiment_ret_20d", "Core_PCE_zscore_60d", "SLB_Schlumberger_ret_1d", "ES_Evergy_ret_1d", "CPB_CampbellSoup_ret_5d", "BTI_BritishAmerican_ret_20d", "EXC_Exelon_ret_1d", "AMD_ret_1d", "T_ret_1d", "GE_ret_1d", "SJM_JM_Smucker_ret_5d"], "is_new": true}, {"model_id": "new_h5_STRESS_LogisticRegression_N25_t4", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 5, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "Core_CPI_zscore_60d", "LLY_zscore_60d", "gjr_condvar_h1", "IBEX_Spain_ret_20d", "PLD_Prologis_ret_5d", "vix_mean_abs_ret_5d", "EOG_EOGResources_ret_5d", "BTI_BritishAmerican_ret_5d", "MSTR_Bitcoin3_ret_1d", "DAX_Germany_zscore_60d", "PPL_PPL_ret_1d", "MRK_Merck_zscore_60d", "PCAR_PaccarInc_ret_5d", "EWA_Australia_zscore_60d", "EWQ_France_zscore_60d", "ES_Evergy_ret_1d", "ASX_Australia_ret_5d", "AMZN_ret_5d", "NVDA_vol_20d", "CPB_CampbellSoup_ret_20d", "CTAS_Cintas_vol_20d", "EWM_Malaysia_ret_1d", "ORCL_vol_20d"], "is_new": true}, {"model_id": "new_h5_STRESS_LogisticRegression_N25_t5", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 5, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "HD_ret_20d", "TM_Telephone_vol_20d", "AMD_ret_1d", "EOG_EOGResources_ret_5d", "EWA_Australia_zscore_60d", "EWM_Malaysia_zscore_60d", "heston_ev_h3", "LOW_Lowes_ret_20d", "EFFR_vol_20d", "EWQ_France_zscore_60d", "INTC_ret_5d", "SPY_zscore_60d", "EWY_Korea_zscore_60d", "MRK_Merck_zscore_60d", "VVIX_ret_20d", "Nikkei_Japan_zscore_60d", "EWQ_France_ret_20d", "US1Y_Rate_ret_5d", "EWJ_Japan_vol_20d", "XOM_ret_20d", "HangSeng_HK_vol_20d", "MS_MorganStanley_ret_5d", "BLK_BlackRock_zscore_60d"], "is_new": true}, {"model_id": "new_h5_STRESS_LogisticRegression_N25_t6", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 5, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "XLK_Tech_zscore_60d", "T10Y2Y_Spread_ret_5d", "DAX_Germany_vol_20d", "INTC_ret_5d", "IWM_SmallCap_vol_20d", "ITT_ITTInc_ret_5d", "ORCL_vol_20d", "CPB_CampbellSoup_vol_20d", "EWA_Australia_zscore_60d", "XLF_Fin_vol_20d", "Nikkei_Japan_vol_20d", "BA_ret_1d", "IBEX_Spain_ret_20d", "PPL_PPL_ret_1d", "hmm_p_stress", "Core_CPI_zscore_60d", "3M_vol_20d", "HD_zscore_60d", "ES_Evergy_ret_1d", "GD_GeneralDynamics_zscore_60d", "DHR_ret_1d", "XLV_Health_zscore_60d", "US1Y_Rate_ret_5d"], "is_new": true}, {"model_id": "new_h5_STRESS_LogisticRegression_N25_t7", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 5, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWG_Germany_ret_20d", "EWS_Singapore_ret_5d", "Industrial_Production_zscore_60d", "Retail_Sales_zscore_60d", "CMCSA_ret_1d", "PAYX_Paychex_vol_20d", "Nikkei_Japan_zscore_60d", "T_ret_1d", "LOW_Lowes_ret_5d", "EWA_Australia_zscore_60d", "SLB_Schlumberger_ret_1d", "GE_ret_1d", "VRP_ma5", "XLV_Health_zscore_60d", "XOM_ret_20d", "AMT_AmericanTower_ret_1d", "T10Y2Y_Spread_ret_5d", "XLY_Disc_vol_20d", "PAYX_Paychex_zscore_60d", "PG_ret_20d", "DIS_vol_20d", "PPL_PPL_ret_1d", "WTI_Oil_FRED_zscore_60d"], "is_new": true}, {"model_id": "new_h5_STRESS_LogisticRegression_N30_t0", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 5, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "PFE_ret_1d", "Brent_Oil_FRED_ret_20d", "HangSeng_HK_ret_1d", "3M_ret_5d", "AMZN_ret_5d", "HD_ret_20d", "MSTR_Bitcoin3_ret_1d", "EXC_Exelon_ret_1d", "BA_ret_1d", "TED_Spread_zscore_60d", "SJM_JM_Smucker_ret_1d", "WTI_Oil_FRED_zscore_60d", "TM_Telephone_vol_20d", "CPB_CampbellSoup_ret_5d", "QQQ_vol_20d", "EWQ_France_ret_20d", "Nikkei_Japan_zscore_60d", "CTAS_Cintas_vol_20d", "SLB_Schlumberger_ret_5d", "TED_Spread_vol_20d", "Industrial_Production_zscore_60d", "PAYX_Paychex_zscore_60d", "JNJ_ret_1d", "DOW_Price_zscore_60d", "GE_ret_1d", "LUV_SouthwestAir_ret_5d", "EMR_Emerson_ret_20d", "Core_CPI_zscore_60d"], "is_new": true}, {"model_id": "new_h5_STRESS_LogisticRegression_N30_t1", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 5, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "CMCSA_ret_1d", "CTAS_Cintas_vol_20d", "SPY_zscore_60d", "AXP_Amex_vol_20d", "TED_Spread_vol_20d", "EOG_EOGResources_ret_5d", "XLB_Materials_zscore_60d", "IWM_SmallCap_vol_20d", "EWY_Korea_zscore_60d", "gjr_condvar_h1", "US30Y_Rate_ret_20d", "INTC_ret_5d", "DE_Deere_vol_20d", "CI_Cigna_vol_20d", "CCI_CrownCastle_vol_20d", "DHR_ret_1d", "vix_acceleration_1d", "LUV_SouthwestAir_ret_5d", "US5Y_Rate_ret_5d", "IBEX_Spain_ret_20d", "NWL_Newell_ret_20d", "EWS_Singapore_ret_5d", "EWA_Australia_ret_1d", "EWY_Korea_ret_20d", "heston_ev_h3", "CPB_CampbellSoup_zscore_60d", "QQQ_vol_20d", "TM_Telephone_ret_1d"], "is_new": true}, {"model_id": "new_h5_STRESS_LogisticRegression_N30_t2", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 5, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "VRP_ma5", "AXP_Amex_ret_20d", "CPB_CampbellSoup_ret_5d", "PLD_Prologis_ret_5d", "spx_momentum_3d", "AMZN_ret_5d", "MSTR_Bitcoin3_ret_1d", "XOM_ret_1d", "BA_ret_1d", "IYM_BasicMaterials_ret_20d", "EWM_Malaysia_vol_20d", "EWC_Canada_zscore_60d", "spx_abs_ret_max_5d", "BLK_BlackRock_zscore_60d", "DAX_Germany_vol_20d", "WTI_Oil_FRED_zscore_60d", "EWQ_France_ret_20d", "MSTR_Bitcoin3_ret_20d", "gjr_condvar_h1", "T10Y2Y_Spread_ret_5d", "NEE_NextEra_ret_20d", "Industrial_Production_zscore_60d", "BTI_BritishAmerican_ret_20d", "EXC_Exelon_zscore_60d", "EWL_Switzerland_vol_20d", "vix_mean_abs_ret_5d", "EWY_Korea_zscore_60d", "Nikkei_Japan_zscore_60d"], "is_new": true}, {"model_id": "new_h5_STRESS_LogisticRegression_N30_t3", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 5, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "IBEX_Spain_ret_20d", "JNJ_ret_1d", "EQR_Equity_ret_1d", "Brent_Oil_FRED_ret_5d", "VOD_Vodafone_zscore_60d", "AVB_AvalonBay_zscore_60d", "US7Y_Rate_ret_20d", "ENB_EnbridgeInc_ret_1d", "MS_MorganStanley_zscore_60d", "QQQ_vol_20d", "DAX_Germany_vol_20d", "ES_Evergy_ret_1d", "INTC_ret_5d", "TM_Telephone_ret_1d", "GILD_Gilead_ret_20d", "AMD_ret_1d", "HUM_Humana_ret_5d", "SCHW_Schwab_ret_5d", "NOC_Northrop_ret_20d", "BDX_Becton_Dickinson_ret_20d", "SLB_Schlumberger_ret_1d", "EWY_Korea_ret_20d", "AMZN_ret_5d", "ORCL_zscore_60d", "spx_abs_ret_max_5d", "IYM_BasicMaterials_ret_20d", "HD_ret_1d", "NVDA_vol_20d"], "is_new": true}, {"model_id": "new_h5_STRESS_LogisticRegression_N30_t4", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 5, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EXC_Exelon_ret_1d", "BLK_BlackRock_zscore_60d", "heston_ev_h3", "PPL_PPL_ret_1d", "ORCL_vol_20d", "SPY_zscore_60d", "LMT_LockheedMartin_ret_1d", "MSTR_Bitcoin3_ret_20d", "GE_ret_1d", "EXC_Exelon_zscore_60d", "VVIX_ret_20d", "DOW_Price_zscore_60d", "INTC_ret_1d", "Industrial_Production_zscore_60d", "US5Y_Rate_ret_5d", "EWM_Malaysia_zscore_60d", "EWY_Korea_ret_20d", "EWY_Korea_zscore_60d", "TGT_Target_zscore_60d", "SJM_JM_Smucker_ret_5d", "heston_var_ev_h5", "US3M_Rate_zscore_60d", "INTC_ret_5d", "HangSeng_HK_vol_20d", "NEE_NextEra_ret_20d", "TXN_vol_20d", "vix_acceleration_1d", "3M_vol_20d"], "is_new": true}, {"model_id": "new_h5_STRESS_LogisticRegression_N30_t5", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 5, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "NWL_Newell_ret_20d", "GILD_Gilead_ret_20d", "MS_MorganStanley_ret_1d", "LOW_Lowes_ret_5d", "EWG_Germany_ret_20d", "BTI_BritishAmerican_ret_5d", "SJM_JM_Smucker_ret_1d", "MRK_Merck_zscore_60d", "XLV_Health_zscore_60d", "Industrial_Production_zscore_60d", "US3M_Rate_vol_20d", "NVDA_vol_20d", "DE_Deere_vol_20d", "QQQ_vol_20d", "FedFunds_zscore_60d", "DAX_Germany_zscore_60d", "EWL_Switzerland_vol_20d", "EWM_Malaysia_zscore_60d", "HangSeng_HK_ret_1d", "DAX_Germany_vol_20d", "EFFR_ret_1d", "TXN_vol_20d", "EWQ_France_zscore_60d", "HD_ret_5d", "EWQ_France_ret_20d", "PPL_PPL_ret_1d", "AMT_AmericanTower_ret_1d", "US7Y_Rate_ret_20d"], "is_new": true}, {"model_id": "new_h5_STRESS_LogisticRegression_N30_t6", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 5, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "XLK_Tech_zscore_60d", "NOC_Northrop_ret_20d", "SO_SouthernCo_ret_5d", "MRK_Merck_zscore_60d", "Retail_Sales_zscore_60d", "LOW_Lowes_ret_20d", "EWM_Malaysia_vol_20d", "VVIX_ret_20d", "BTI_BritishAmerican_ret_20d", "MSTR_Bitcoin3_ret_20d", "XLF_Fin_vol_20d", "CPB_CampbellSoup_vol_20d", "EWA_Australia_ret_1d", "FedFunds_zscore_60d", "heston_ev_h3", "US1Y_Rate_ret_20d", "gjr_condvar_h1", "EWM_Malaysia_zscore_60d", "EWQ_France_ret_20d", "DAX_Germany_zscore_60d", "SBUX_ret_5d", "MS_MorganStanley_ret_5d", "MS_MorganStanley_zscore_60d", "CLX_Clorox_vol_20d", "MSTR_Bitcoin3_ret_1d", "EWL_Switzerland_vol_20d", "SJM_JM_Smucker_ret_5d", "HD_ret_1d"], "is_new": true}, {"model_id": "new_h5_STRESS_LogisticRegression_N30_t7", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 5, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "DOW_Price_zscore_60d", "vix_mean_abs_ret_5d", "Industrial_Production_zscore_60d", "Core_PCE_zscore_60d", "EOG_EOGResources_vol_20d", "IWM_SmallCap_vol_20d", "SBUX_ret_5d", "HangSeng_HK_vol_20d", "SPY_zscore_60d", "spx_momentum_3d", "LMT_LockheedMartin_ret_1d", "EWH_HongKong_ret_5d", "EQIX_Equinix_ret_5d", "heston_var_ev_h5", "Nikkei_Japan_zscore_60d", "BA_ret_1d", "MSTR_Bitcoin3_ret_1d", "EWM_Malaysia_vol_20d", "SO_SouthernCo_ret_5d", "CPB_CampbellSoup_zscore_60d", "TED_Spread_vol_20d", "3M_vol_20d", "VRP_ma5", "IYM_BasicMaterials_ret_20d", "gjr_condvar_h1", "GE_ret_1d", "spx_abs_ret_max_5d", "ASX_Australia_ret_5d"], "is_new": true}, {"model_id": "new_h5_GLOBAL_XGBoost_N5_t0", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 5, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "BA_ret_1d", "BDX_Becton_Dickinson_ret_20d", "SBUX_vol_20d"], "is_new": true}, {"model_id": "new_h5_GLOBAL_XGBoost_N5_t1", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 5, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "3M_vol_20d", "M_Macys_vol_20d", "CPB_CampbellSoup_ret_5d"], "is_new": true}, {"model_id": "new_h5_GLOBAL_XGBoost_N5_t2", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 5, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "SLB_Schlumberger_ret_5d", "DE_Deere_vol_20d", "ASX_Australia_ret_5d"], "is_new": true}, {"model_id": "new_h5_GLOBAL_XGBoost_N5_t3", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 5, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWQ_France_zscore_60d", "HangSeng_HK_ret_1d", "US30Y_Rate_ret_20d"], "is_new": true}, {"model_id": "new_h5_GLOBAL_XGBoost_N5_t4", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 5, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "US1Y_Rate_ret_5d", "CTAS_Cintas_vol_20d", "AMD_ret_5d"], "is_new": true}, {"model_id": "new_h5_GLOBAL_XGBoost_N5_t5", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 5, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EMR_Emerson_ret_20d", "EXC_Exelon_zscore_60d", "3M_ret_5d"], "is_new": true}, {"model_id": "new_h5_GLOBAL_XGBoost_N5_t6", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 5, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "US30Y_Rate_ret_20d", "CI_Cigna_vol_20d", "INTC_ret_5d"], "is_new": true}, {"model_id": "new_h5_GLOBAL_XGBoost_N5_t7", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 5, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "BTI_BritishAmerican_ret_20d", "DHR_vol_20d", "EXC_Exelon_zscore_60d"], "is_new": true}, {"model_id": "new_h5_GLOBAL_XGBoost_N8_t0", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 5, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "AMD_ret_1d", "3M_vol_20d", "XLY_Disc_vol_20d", "HD_ret_5d", "HangSeng_HK_ret_1d", "EWS_Singapore_ret_5d"], "is_new": true}, {"model_id": "new_h5_GLOBAL_XGBoost_N8_t1", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 5, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "Core_CPI_zscore_60d", "US7Y_Rate_ret_20d", "MRK_Merck_zscore_60d", "TED_Spread_zscore_60d", "PLD_Prologis_ret_5d", "EWM_Malaysia_ret_1d"], "is_new": true}, {"model_id": "new_h5_GLOBAL_XGBoost_N8_t2", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 5, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "FedFunds_zscore_60d", "AVB_AvalonBay_zscore_60d", "MRK_Merck_zscore_60d", "CCI_CrownCastle_vol_20d", "EWG_Germany_ret_20d", "ASX_Australia_vol_20d"], "is_new": true}, {"model_id": "new_h5_GLOBAL_XGBoost_N8_t3", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 5, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWM_Malaysia_zscore_60d", "US3M_Rate_vol_20d", "MRK_Merck_zscore_60d", "EWG_Germany_vol_20d", "ES_Evergy_ret_1d", "PAYX_Paychex_zscore_60d"], "is_new": true}, {"model_id": "new_h5_GLOBAL_XGBoost_N8_t4", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 5, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWL_Switzerland_zscore_60d", "DAX_Germany_vol_20d", "NEE_NextEra_ret_20d", "SBUX_vol_20d", "Core_PCE_zscore_60d", "NVDA_vol_20d"], "is_new": true}, {"model_id": "new_h5_GLOBAL_XGBoost_N8_t5", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 5, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "IYR_US_REIT2_zscore_60d", "HangSeng_HK_vol_20d", "US1Y_Rate_ret_5d", "CMCSA_ret_1d", "HD_ret_5d", "HUM_Humana_ret_5d"], "is_new": true}, {"model_id": "new_h5_GLOBAL_XGBoost_N8_t6", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 5, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EFFR_ret_1d", "vix_acceleration_1d", "CPB_CampbellSoup_ret_5d", "INTC_ret_1d", "Michigan_Sentiment_ret_20d", "AXP_Amex_vol_20d"], "is_new": true}, {"model_id": "new_h5_GLOBAL_XGBoost_N8_t7", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 5, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "SCHW_Schwab_ret_5d", "HangSeng_HK_vol_20d", "MS_MorganStanley_zscore_60d", "TED_Spread_vol_20d", "Retail_Sales_zscore_60d", "US3M_Rate_zscore_60d"], "is_new": true}, {"model_id": "new_h5_GLOBAL_XGBoost_N10_t0", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 5, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "LMT_LockheedMartin_vol_20d", "Brent_Oil_FRED_ret_20d", "EWM_Malaysia_vol_20d", "PLD_Prologis_ret_5d", "BA_ret_1d", "EWH_HongKong_ret_5d", "LMT_LockheedMartin_ret_1d", "US6M_Rate_ret_20d"], "is_new": true}, {"model_id": "new_h5_GLOBAL_XGBoost_N10_t1", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 5, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "SLB_Schlumberger_ret_1d", "heston_ev_h3", "HD_zscore_60d", "CPB_CampbellSoup_ret_5d", "PFE_ret_1d", "SO_SouthernCo_ret_5d", "NVDA_vol_20d", "PPL_PPL_ret_1d"], "is_new": true}, {"model_id": "new_h5_GLOBAL_XGBoost_N10_t2", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 5, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "VVIX_ret_20d", "EWL_Switzerland_zscore_60d", "US3Y_Rate_ret_5d", "HangSeng_HK_ret_5d", "VOD_Vodafone_zscore_60d", "spx_vol_5d", "IYM_BasicMaterials_ret_20d", "SBUX_zscore_60d"], "is_new": true}, {"model_id": "new_h5_GLOBAL_XGBoost_N10_t3", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 5, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWG_Germany_vol_20d", "INTC_ret_5d", "EMR_Emerson_ret_20d", "DAX_Germany_vol_20d", "BLK_BlackRock_zscore_60d", "HangSeng_HK_ret_5d", "TM_Telephone_vol_20d", "CPB_CampbellSoup_ret_20d"], "is_new": true}, {"model_id": "new_h5_GLOBAL_XGBoost_N10_t4", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 5, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWS_Singapore_ret_5d", "TGT_Target_zscore_60d", "VOD_Vodafone_zscore_60d", "EWM_Malaysia_vol_20d", "Brent_Oil_FRED_ret_20d", "EWJ_Japan_vol_20d", "TED_Spread_vol_20d", "Nikkei_Japan_zscore_60d"], "is_new": true}, {"model_id": "new_h5_GLOBAL_XGBoost_N10_t5", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 5, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "ITT_ITTInc_ret_5d", "DE_Deere_vol_20d", "HD_ret_1d", "XLF_Fin_vol_20d", "EWC_Canada_zscore_60d", "SBUX_vol_20d", "CTAS_Cintas_vol_20d", "AXP_Amex_ret_20d"], "is_new": true}, {"model_id": "new_h5_GLOBAL_XGBoost_N10_t6", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 5, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "ES_Evergy_ret_1d", "US1Y_Rate_ret_5d", "IYR_US_REIT2_zscore_60d", "CPB_CampbellSoup_vol_20d", "3M_vol_20d", "EWY_Korea_ret_20d", "hmm_p_stress", "Brent_Oil_FRED_ret_20d"], "is_new": true}, {"model_id": "new_h5_GLOBAL_XGBoost_N10_t7", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 5, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "AMZN_ret_5d", "EWM_Malaysia_vol_20d", "XLV_Health_zscore_60d", "PLD_Prologis_ret_5d", "spx_momentum_3d", "M_Macys_vol_20d", "DE_Deere_ret_5d", "Core_CPI_zscore_60d"], "is_new": true}, {"model_id": "new_h5_GLOBAL_XGBoost_N12_t0", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 5, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "GILD_Gilead_ret_20d", "EWS_Singapore_ret_5d", "SJM_JM_Smucker_ret_5d", "VOD_Vodafone_zscore_60d", "CCI_CrownCastle_vol_20d", "CPB_CampbellSoup_zscore_60d", "HangSeng_HK_ret_5d", "heston_var_ev_h7", "SO_SouthernCo_ret_5d", "DIS_vol_20d"], "is_new": true}, {"model_id": "new_h5_GLOBAL_XGBoost_N12_t1", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 5, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "MSTR_Bitcoin3_ret_1d", "DIS_vol_20d", "EWY_Korea_zscore_60d", "HD_ret_5d", "BTI_BritishAmerican_ret_20d", "TED_Spread_zscore_60d", "INTC_ret_1d", "AMT_AmericanTower_ret_1d", "BTI_BritishAmerican_ret_5d", "vix_mean_abs_ret_5d"], "is_new": true}, {"model_id": "new_h5_GLOBAL_XGBoost_N12_t2", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 5, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "JNJ_ret_1d", "AMGN_Amgen_ret_1d", "XLB_Materials_zscore_60d", "XLK_Tech_zscore_60d", "MO_AltriaMG_ret_1d", "SCHW_Schwab_ret_5d", "IYM_BasicMaterials_ret_20d", "MSTR_Bitcoin3_ret_5d", "TED_Spread_vol_20d", "CLX_Clorox_vol_20d"], "is_new": true}, {"model_id": "new_h5_GLOBAL_XGBoost_N12_t3", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 5, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "Brent_Oil_FRED_ret_5d", "Industrial_Production_zscore_60d", "EWM_Malaysia_ret_1d", "ES_Evergy_ret_1d", "WTI_Oil_FRED_zscore_60d", "EWL_Switzerland_zscore_60d", "CPB_CampbellSoup_ret_20d", "EXC_Exelon_zscore_60d", "US7Y_Rate_ret_20d", "EWM_Malaysia_zscore_60d"], "is_new": true}, {"model_id": "new_h5_GLOBAL_XGBoost_N12_t4", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 5, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWH_HongKong_ret_5d", "LUV_SouthwestAir_ret_5d", "XLK_Tech_zscore_60d", "M_Macys_vol_20d", "EWS_Singapore_ret_5d", "GILD_Gilead_ret_20d", "Retail_Sales_zscore_60d", "ENB_EnbridgeInc_ret_1d", "Nikkei_Japan_vol_20d", "GE_ret_1d"], "is_new": true}, {"model_id": "new_h5_GLOBAL_XGBoost_N12_t5", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 5, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "NVDA_vol_20d", "INTC_ret_5d", "ENB_EnbridgeInc_ret_1d", "PAYX_Paychex_ret_20d", "EWC_Canada_zscore_60d", "3M_vol_20d", "TXN_vol_20d", "QQQ_vol_20d", "DE_Deere_vol_20d", "EWY_Korea_zscore_60d"], "is_new": true}, {"model_id": "new_h5_GLOBAL_XGBoost_N12_t6", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 5, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "TED_Spread_vol_20d", "AVB_AvalonBay_zscore_60d", "AMD_ret_5d", "ENB_EnbridgeInc_ret_1d", "vix_acceleration_1d", "LMT_LockheedMartin_vol_20d", "spx_vol_5d", "HD_ret_5d", "AMD_ret_1d", "Nikkei_Japan_zscore_60d"], "is_new": true}, {"model_id": "new_h5_GLOBAL_XGBoost_N12_t7", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 5, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "NWL_Newell_ret_20d", "LUV_SouthwestAir_ret_5d", "BTI_BritishAmerican_ret_20d", "SBUX_zscore_60d", "DIS_vol_20d", "SBUX_ret_5d", "SJM_JM_Smucker_ret_1d", "INTC_ret_1d", "EWL_Switzerland_zscore_60d", "EFFR_vol_20d"], "is_new": true}, {"model_id": "new_h5_GLOBAL_XGBoost_N15_t0", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 5, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "heston_ev_h3", "MSTR_Bitcoin3_ret_5d", "IWM_SmallCap_vol_20d", "IYR_US_REIT2_zscore_60d", "ORCL_vol_20d", "HD_zscore_60d", "EXC_Exelon_zscore_60d", "CPB_CampbellSoup_zscore_60d", "VOD_Vodafone_zscore_60d", "EWC_Canada_zscore_60d", "3M_vol_20d", "AORD_AUS_zscore_60d", "Brent_Oil_FRED_ret_20d"], "is_new": true}, {"model_id": "new_h5_GLOBAL_XGBoost_N15_t1", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 5, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "US30Y_Rate_ret_20d", "EQR_Equity_ret_1d", "EXC_Exelon_ret_1d", "AMD_ret_5d", "EWQ_France_ret_20d", "AXP_Amex_vol_20d", "US7Y_Rate_ret_20d", "ENB_EnbridgeInc_ret_1d", "DHR_ret_1d", "SLB_Schlumberger_ret_1d", "EWM_Malaysia_ret_1d", "IBEX_Spain_ret_20d", "EWG_Germany_ret_20d"], "is_new": true}, {"model_id": "new_h5_GLOBAL_XGBoost_N15_t2", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 5, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "MSTR_Bitcoin3_ret_5d", "NWL_Newell_ret_20d", "EWG_Germany_ret_20d", "AMT_AmericanTower_ret_1d", "VRP_ma5", "hmm_p_stress", "AXP_Amex_ret_20d", "MSTR_Bitcoin3_ret_1d", "Nikkei_Japan_vol_20d", "US3Y_Rate_ret_5d", "INTC_ret_1d", "T_ret_1d", "BA_ret_1d"], "is_new": true}, {"model_id": "new_h5_GLOBAL_XGBoost_N15_t3", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 5, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "MS_MorganStanley_ret_5d", "NEE_NextEra_ret_20d", "HangSeng_HK_ret_5d", "IBEX_Spain_ret_20d", "MO_AltriaMG_ret_1d", "BTI_BritishAmerican_ret_5d", "IYM_BasicMaterials_ret_20d", "PG_ret_20d", "CPB_CampbellSoup_ret_20d", "SBUX_vol_20d", "spx_vol_5d", "US5Y_Rate_ret_5d", "DOW_Price_zscore_60d"], "is_new": true}, {"model_id": "new_h5_GLOBAL_XGBoost_N15_t4", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 5, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "DHR_ret_1d", "TED_Spread_vol_20d", "SJM_JM_Smucker_ret_5d", "EWG_Germany_ret_20d", "HD_ret_1d", "US5Y_Rate_ret_5d", "spx_momentum_3d", "DAX_Germany_zscore_60d", "TM_Telephone_ret_1d", "GD_GeneralDynamics_zscore_60d", "HUM_Humana_ret_5d", "IYM_BasicMaterials_ret_20d", "IWM_SmallCap_vol_20d"], "is_new": true}, {"model_id": "new_h5_GLOBAL_XGBoost_N15_t5", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 5, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "XLY_Disc_vol_20d", "SO_SouthernCo_ret_5d", "DHR_vol_20d", "TXN_vol_20d", "AMT_AmericanTower_ret_1d", "EWC_Canada_zscore_60d", "SBUX_vol_20d", "US6M_Rate_ret_20d", "GILD_Gilead_ret_20d", "GD_GeneralDynamics_zscore_60d", "AORD_AUS_zscore_60d", "BA_ret_1d", "AMD_ret_1d"], "is_new": true}, {"model_id": "new_h5_GLOBAL_XGBoost_N15_t6", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 5, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "TED_Spread_zscore_60d", "US5Y_Rate_ret_5d", "SBUX_ret_5d", "QQQ_vol_20d", "VVIX_ret_20d", "AXP_Amex_vol_20d", "EWM_Malaysia_ret_1d", "Nikkei_Japan_zscore_60d", "EXC_Exelon_ret_1d", "XLF_Fin_vol_20d", "MSTR_Bitcoin3_ret_1d", "HangSeng_HK_ret_5d", "DE_Deere_vol_20d"], "is_new": true}, {"model_id": "new_h5_GLOBAL_XGBoost_N15_t7", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 5, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "US7Y_Rate_ret_20d", "NEE_NextEra_ret_20d", "T10Y2Y_Spread_ret_5d", "XLF_Fin_vol_20d", "EOG_EOGResources_ret_5d", "SJM_JM_Smucker_ret_5d", "Nikkei_Japan_zscore_60d", "US3M_Rate_vol_20d", "SBUX_vol_20d", "PCAR_PaccarInc_ret_5d", "US6M_Rate_ret_20d", "EWM_Malaysia_zscore_60d", "HUM_Humana_ret_5d"], "is_new": true}, {"model_id": "new_h5_GLOBAL_XGBoost_N20_t0", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 5, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "LOW_Lowes_ret_5d", "QQQ_vol_20d", "DAX_Germany_zscore_60d", "vix_acceleration_1d", "EWG_Germany_ret_20d", "CI_Cigna_vol_20d", "INTC_ret_1d", "Michigan_Sentiment_ret_20d", "ASX_Australia_ret_5d", "PFE_ret_1d", "VRP_ma5", "HD_ret_5d", "T_ret_1d", "BTI_BritishAmerican_ret_5d", "INTC_ret_5d", "TED_Spread_zscore_60d", "Core_PCE_zscore_60d", "AMZN_ret_5d"], "is_new": true}, {"model_id": "new_h5_GLOBAL_XGBoost_N20_t1", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 5, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "HangSeng_HK_vol_20d", "PFE_ret_1d", "LOW_Lowes_ret_5d", "BTI_BritishAmerican_ret_20d", "EQIX_Equinix_ret_5d", "EXC_Exelon_zscore_60d", "MRK_Merck_zscore_60d", "SBUX_zscore_60d", "EWQ_France_ret_20d", "Retail_Sales_zscore_60d", "US3M_Rate_zscore_60d", "TM_Telephone_vol_20d", "GD_GeneralDynamics_zscore_60d", "SLB_Schlumberger_ret_5d", "ASX_Australia_vol_20d", "BDX_Becton_Dickinson_ret_20d", "DIS_vol_20d", "3M_vol_20d"], "is_new": true}, {"model_id": "new_h5_GLOBAL_XGBoost_N20_t2", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 5, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "heston_var_ev_h5", "BA_ret_1d", "PAYX_Paychex_ret_20d", "T10Y2Y_Spread_ret_5d", "US5Y_Rate_ret_5d", "MSTR_Bitcoin3_ret_5d", "BTI_BritishAmerican_ret_5d", "Retail_Sales_zscore_60d", "TED_Spread_vol_20d", "EWQ_France_ret_20d", "DOW_Price_zscore_60d", "spx_vol_5d", "DE_Deere_vol_20d", "MRK_Merck_zscore_60d", "vix_acceleration_1d", "BDX_Becton_Dickinson_ret_20d", "WTI_Oil_FRED_zscore_60d", "CPB_CampbellSoup_vol_20d"], "is_new": true}, {"model_id": "new_h5_GLOBAL_XGBoost_N20_t3", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 5, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EFFR_vol_20d", "EQIX_Equinix_ret_5d", "DIS_vol_20d", "ES_Evergy_ret_1d", "EWS_Singapore_ret_5d", "DE_Deere_vol_20d", "M_Macys_vol_20d", "ASX_Australia_vol_20d", "CMCSA_ret_1d", "PCAR_PaccarInc_ret_5d", "LOW_Lowes_ret_20d", "HD_ret_5d", "DHR_vol_20d", "MS_MorganStanley_ret_5d", "XOM_ret_1d", "TED_Spread_zscore_60d", "T10Y2Y_Spread_ret_5d", "EWG_Germany_vol_20d"], "is_new": true}, {"model_id": "new_h5_GLOBAL_XGBoost_N20_t4", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 5, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "GE_ret_1d", "DOW_Price_zscore_60d", "EWH_HongKong_ret_5d", "IBEX_Spain_ret_20d", "US1Y_Rate_ret_20d", "EWA_Australia_ret_1d", "spx_abs_ret_max_5d", "XLY_Disc_vol_20d", "heston_var_ev_h7", "ASX_Australia_vol_20d", "NVDA_vol_20d", "CCI_CrownCastle_vol_20d", "SPY_zscore_60d", "IYM_BasicMaterials_ret_20d", "Michigan_Sentiment_ret_20d", "SBUX_ret_5d", "DHR_vol_20d", "ORCL_zscore_60d"], "is_new": true}, {"model_id": "new_h5_GLOBAL_XGBoost_N20_t5", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 5, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "AMT_AmericanTower_ret_1d", "DE_Deere_vol_20d", "EWH_HongKong_ret_5d", "spx_vol_5d", "VRP_ma5", "HD_zscore_60d", "US1Y_Rate_ret_5d", "Retail_Sales_zscore_60d", "PAYX_Paychex_ret_20d", "CCI_CrownCastle_vol_20d", "GD_GeneralDynamics_zscore_60d", "MS_MorganStanley_ret_5d", "PPL_PPL_ret_1d", "EWM_Malaysia_zscore_60d", "EWG_Germany_vol_20d", "heston_var_ev_h7", "VOD_Vodafone_zscore_60d", "AXP_Amex_vol_20d"], "is_new": true}, {"model_id": "new_h5_GLOBAL_XGBoost_N20_t6", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 5, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "PAYX_Paychex_ret_20d", "AVB_AvalonBay_zscore_60d", "CTAS_Cintas_vol_20d", "T10Y2Y_Spread_ret_5d", "EWJ_Japan_vol_20d", "EFFR_vol_20d", "SBUX_vol_20d", "FedFunds_zscore_60d", "AMD_ret_1d", "EWA_Australia_zscore_60d", "Core_CPI_zscore_60d", "ORCL_vol_20d", "VRP_ma5", "HD_ret_5d", "heston_var_ev_h3", "Retail_Sales_zscore_60d", "GD_GeneralDynamics_zscore_60d", "EWQ_France_zscore_60d"], "is_new": true}, {"model_id": "new_h5_GLOBAL_XGBoost_N20_t7", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 5, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "gjr_condvar_h1", "PAYX_Paychex_vol_20d", "hmm_p_stress", "Nikkei_Japan_vol_20d", "MRK_Merck_zscore_60d", "CI_Cigna_vol_20d", "BLK_BlackRock_zscore_60d", "DHR_vol_20d", "CPB_CampbellSoup_ret_20d", "SLB_Schlumberger_ret_5d", "M_Macys_vol_20d", "EXC_Exelon_zscore_60d", "JNJ_ret_1d", "XLF_Fin_vol_20d", "US30Y_Rate_ret_20d", "DHR_ret_1d", "CLX_Clorox_vol_20d", "US5Y_Rate_ret_5d"], "is_new": true}, {"model_id": "new_h5_GLOBAL_XGBoost_N25_t0", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 5, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "QQQ_vol_20d", "NEE_NextEra_ret_20d", "EWS_Singapore_ret_5d", "SLB_Schlumberger_ret_5d", "vix_acceleration_1d", "CLX_Clorox_vol_20d", "US1Y_Rate_ret_20d", "XLK_Tech_zscore_60d", "AMZN_ret_5d", "T_ret_1d", "LMT_LockheedMartin_ret_1d", "HangSeng_HK_ret_5d", "AMD_ret_1d", "EWM_Malaysia_ret_1d", "BLK_BlackRock_zscore_60d", "EWA_Australia_zscore_60d", "DHR_vol_20d", "LOW_Lowes_ret_20d", "EWL_Switzerland_zscore_60d", "MO_AltriaMG_ret_1d", "ITT_ITTInc_ret_5d", "IYM_BasicMaterials_ret_20d", "LOW_Lowes_ret_5d"], "is_new": true}, {"model_id": "new_h5_GLOBAL_XGBoost_N25_t1", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 5, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "IWM_SmallCap_vol_20d", "NFCI_ret_5d", "AMGN_Amgen_ret_1d", "3M_ret_5d", "heston_ev_h3", "NOC_Northrop_ret_20d", "HUM_Humana_ret_5d", "DHR_ret_1d", "spx_momentum_3d", "XLF_Fin_vol_20d", "EOG_EOGResources_vol_20d", "DHR_vol_20d", "LOW_Lowes_ret_5d", "US3M_Rate_zscore_60d", "SBUX_zscore_60d", "EWY_Korea_ret_20d", "hmm_p_stress", "PG_ret_20d", "AXP_Amex_vol_20d", "PAYX_Paychex_vol_20d", "CLX_Clorox_vol_20d", "AVB_AvalonBay_zscore_60d", "MS_MorganStanley_ret_1d"], "is_new": true}, {"model_id": "new_h5_GLOBAL_XGBoost_N25_t2", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 5, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWM_Malaysia_zscore_60d", "gjr_condvar_h1", "3M_vol_20d", "EWM_Malaysia_ret_1d", "DE_Deere_vol_20d", "EMR_Emerson_ret_20d", "EWJ_Japan_vol_20d", "EFFR_vol_20d", "EWY_Korea_ret_20d", "LMT_LockheedMartin_vol_20d", "HangSeng_HK_vol_20d", "spx_momentum_3d", "EWC_Canada_zscore_60d", "SCHW_Schwab_ret_5d", "EXC_Exelon_zscore_60d", "INTC_ret_5d", "EWH_HongKong_ret_5d", "Nikkei_Japan_zscore_60d", "IYR_US_REIT2_zscore_60d", "M_Macys_vol_20d", "ASX_Australia_vol_20d", "DHR_ret_1d", "MS_MorganStanley_ret_1d"], "is_new": true}, {"model_id": "new_h5_GLOBAL_XGBoost_N25_t3", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 5, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "CCI_CrownCastle_vol_20d", "MS_MorganStanley_zscore_60d", "EWL_Switzerland_zscore_60d", "SLB_Schlumberger_ret_5d", "ENB_EnbridgeInc_ret_1d", "EWY_Korea_zscore_60d", "INTC_ret_1d", "US7Y_Rate_ret_20d", "MSTR_Bitcoin3_ret_20d", "ES_Evergy_ret_1d", "spx_abs_ret_max_5d", "EWM_Malaysia_vol_20d", "XLY_Disc_vol_20d", "SBUX_vol_20d", "DHR_vol_20d", "HD_ret_1d", "TM_Telephone_ret_1d", "FedFunds_zscore_60d", "EWL_Switzerland_vol_20d", "US3Y_Rate_ret_5d", "Nikkei_Japan_vol_20d", "AXP_Amex_vol_20d", "heston_var_ev_h3"], "is_new": true}, {"model_id": "new_h5_GLOBAL_XGBoost_N25_t4", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 5, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "US6M_Rate_ret_20d", "CCI_CrownCastle_vol_20d", "3M_vol_20d", "LMT_LockheedMartin_ret_1d", "CPB_CampbellSoup_ret_20d", "LMT_LockheedMartin_vol_20d", "gjr_condvar_h1", "AORD_AUS_zscore_60d", "PFE_ret_1d", "BTI_BritishAmerican_ret_5d", "XOM_ret_20d", "BLK_BlackRock_zscore_60d", "TED_Spread_zscore_60d", "Brent_Oil_FRED_ret_5d", "NFCI_ret_5d", "MO_AltriaMG_ret_1d", "EWG_Germany_ret_20d", "JNJ_ret_1d", "US5Y_Rate_ret_5d", "TGT_Target_zscore_60d", "LOW_Lowes_ret_5d", "BTI_BritishAmerican_ret_20d", "LUV_SouthwestAir_ret_5d"], "is_new": true}, {"model_id": "new_h5_GLOBAL_XGBoost_N25_t5", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 5, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "XLV_Health_zscore_60d", "EOG_EOGResources_ret_5d", "US1Y_Rate_ret_5d", "EWM_Malaysia_vol_20d", "spx_abs_ret_max_5d", "MRK_Merck_zscore_60d", "EWQ_France_ret_20d", "GE_ret_1d", "DOW_Price_zscore_60d", "GILD_Gilead_ret_20d", "XLB_Materials_zscore_60d", "IYM_BasicMaterials_ret_20d", "AMD_ret_1d", "CCI_CrownCastle_vol_20d", "INTC_ret_5d", "vix_acceleration_1d", "TGT_Target_zscore_60d", "AMD_ret_5d", "BA_ret_1d", "NOC_Northrop_ret_20d", "FedFunds_zscore_60d", "TED_Spread_vol_20d", "NEE_NextEra_ret_20d"], "is_new": true}, {"model_id": "new_h5_GLOBAL_XGBoost_N25_t6", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 5, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "VOD_Vodafone_zscore_60d", "US6M_Rate_ret_20d", "EWJ_Japan_vol_20d", "ES_Evergy_ret_1d", "GD_GeneralDynamics_zscore_60d", "Michigan_Sentiment_ret_20d", "XOM_ret_1d", "M_Macys_vol_20d", "spx_abs_ret_max_5d", "XLB_Materials_zscore_60d", "EOG_EOGResources_ret_5d", "T10Y2Y_Spread_ret_5d", "vix_mean_abs_ret_5d", "Nikkei_Japan_zscore_60d", "EWG_Germany_ret_20d", "TED_Spread_zscore_60d", "US3M_Rate_vol_20d", "US5Y_Rate_ret_5d", "NFCI_ret_5d", "SLB_Schlumberger_ret_5d", "AMD_ret_1d", "CPB_CampbellSoup_vol_20d", "spx_momentum_3d"], "is_new": true}, {"model_id": "new_h5_GLOBAL_XGBoost_N25_t7", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 5, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "BA_ret_1d", "ASX_Australia_ret_5d", "INTC_ret_5d", "IBEX_Spain_ret_20d", "PAYX_Paychex_vol_20d", "Michigan_Sentiment_ret_20d", "HD_ret_1d", "EOG_EOGResources_ret_5d", "HD_ret_5d", "XLY_Disc_vol_20d", "GE_ret_1d", "HangSeng_HK_vol_20d", "DHR_vol_20d", "US30Y_Rate_ret_20d", "EQIX_Equinix_ret_5d", "TM_Telephone_vol_20d", "CMCSA_ret_1d", "SO_SouthernCo_ret_5d", "EWA_Australia_zscore_60d", "EFFR_ret_1d", "Industrial_Production_zscore_60d", "LUV_SouthwestAir_ret_5d", "CI_Cigna_vol_20d"], "is_new": true}, {"model_id": "new_h5_GLOBAL_XGBoost_N30_t0", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 5, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "SBUX_ret_5d", "heston_var_ev_h7", "TM_Telephone_ret_1d", "EQR_Equity_ret_1d", "ORCL_zscore_60d", "EFFR_vol_20d", "XLF_Fin_vol_20d", "PFE_ret_1d", "ES_Evergy_ret_1d", "CTAS_Cintas_vol_20d", "EWQ_France_zscore_60d", "ORCL_vol_20d", "XLV_Health_zscore_60d", "AMT_AmericanTower_ret_1d", "US1Y_Rate_ret_5d", "MS_MorganStanley_ret_5d", "Nikkei_Japan_vol_20d", "AMGN_Amgen_ret_1d", "EOG_EOGResources_ret_5d", "DE_Deere_ret_5d", "VOD_Vodafone_zscore_60d", "M_Macys_vol_20d", "EWM_Malaysia_vol_20d", "Brent_Oil_FRED_ret_20d", "NFCI_ret_5d", "LOW_Lowes_ret_20d", "AXP_Amex_vol_20d", "Core_CPI_zscore_60d"], "is_new": true}, {"model_id": "new_h5_GLOBAL_XGBoost_N30_t1", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 5, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "HUM_Humana_ret_5d", "vix_acceleration_1d", "AMD_ret_1d", "EWH_HongKong_ret_5d", "IBEX_Spain_ret_20d", "CPB_CampbellSoup_ret_20d", "Retail_Sales_zscore_60d", "NOC_Northrop_ret_20d", "3M_vol_20d", "XLV_Health_zscore_60d", "SBUX_vol_20d", "Core_CPI_zscore_60d", "EWG_Germany_ret_20d", "SJM_JM_Smucker_ret_1d", "heston_var_ev_h7", "heston_ev_h3", "EWL_Switzerland_vol_20d", "T10Y2Y_Spread_ret_5d", "MSTR_Bitcoin3_ret_1d", "HangSeng_HK_ret_1d", "AMT_AmericanTower_ret_1d", "FedFunds_zscore_60d", "PLD_Prologis_ret_5d", "AORD_AUS_zscore_60d", "GE_ret_1d", "DOW_Price_zscore_60d", "AXP_Amex_ret_20d", "TXN_vol_20d"], "is_new": true}, {"model_id": "new_h5_GLOBAL_XGBoost_N30_t2", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 5, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "TED_Spread_zscore_60d", "NVDA_vol_20d", "EWA_Australia_ret_1d", "LUV_SouthwestAir_ret_5d", "VRP_ma5", "GD_GeneralDynamics_zscore_60d", "EWY_Korea_ret_20d", "AMD_ret_5d", "Brent_Oil_FRED_ret_5d", "DOW_Price_zscore_60d", "DE_Deere_ret_5d", "MS_MorganStanley_ret_1d", "XLB_Materials_zscore_60d", "HangSeng_HK_ret_5d", "Core_PCE_zscore_60d", "IWM_SmallCap_vol_20d", "EWS_Singapore_ret_5d", "ITT_ITTInc_ret_5d", "EWY_Korea_zscore_60d", "EWC_Canada_zscore_60d", "DAX_Germany_vol_20d", "SLB_Schlumberger_ret_1d", "PPL_PPL_ret_1d", "heston_var_ev_h5", "LLY_zscore_60d", "AVB_AvalonBay_zscore_60d", "T_ret_1d", "EMR_Emerson_ret_20d"], "is_new": true}, {"model_id": "new_h5_GLOBAL_XGBoost_N30_t3", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 5, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "MRK_Merck_zscore_60d", "XLY_Disc_vol_20d", "EWL_Switzerland_zscore_60d", "EOG_EOGResources_vol_20d", "EQR_Equity_ret_1d", "GILD_Gilead_ret_20d", "AMD_ret_1d", "T10Y2Y_Spread_ret_5d", "HUM_Humana_ret_5d", "heston_ev_h3", "AMGN_Amgen_ret_1d", "SJM_JM_Smucker_ret_1d", "heston_var_ev_h3", "US3Y_Rate_ret_5d", "EWC_Canada_zscore_60d", "TED_Spread_zscore_60d", "EWS_Singapore_ret_5d", "FedFunds_zscore_60d", "CCI_CrownCastle_vol_20d", "NEE_NextEra_ret_20d", "DHR_ret_1d", "XLF_Fin_vol_20d", "DIS_vol_20d", "ITT_ITTInc_ret_5d", "DE_Deere_vol_20d", "SLB_Schlumberger_ret_5d", "XOM_ret_20d", "hmm_p_stress"], "is_new": true}, {"model_id": "new_h5_GLOBAL_XGBoost_N30_t4", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 5, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWC_Canada_zscore_60d", "EWH_HongKong_ret_5d", "DE_Deere_vol_20d", "DOW_Price_zscore_60d", "Retail_Sales_zscore_60d", "NWL_Newell_ret_20d", "AMT_AmericanTower_ret_1d", "EWG_Germany_vol_20d", "VOD_Vodafone_zscore_60d", "EQR_Equity_ret_1d", "DAX_Germany_vol_20d", "US3M_Rate_vol_20d", "CPB_CampbellSoup_vol_20d", "NFCI_ret_5d", "IWM_SmallCap_vol_20d", "SJM_JM_Smucker_ret_5d", "ASX_Australia_vol_20d", "MS_MorganStanley_ret_1d", "EXC_Exelon_zscore_60d", "EWJ_Japan_vol_20d", "SO_SouthernCo_ret_5d", "MRK_Merck_zscore_60d", "IYM_BasicMaterials_ret_20d", "BTI_BritishAmerican_ret_20d", "DHR_ret_1d", "SLB_Schlumberger_ret_1d", "hmm_p_stress", "Michigan_Sentiment_ret_20d"], "is_new": true}, {"model_id": "new_h5_GLOBAL_XGBoost_N30_t5", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 5, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "ITT_ITTInc_ret_5d", "SLB_Schlumberger_ret_5d", "EWS_Singapore_ret_5d", "AMD_ret_1d", "XLV_Health_zscore_60d", "PAYX_Paychex_zscore_60d", "MS_MorganStanley_ret_5d", "US7Y_Rate_ret_20d", "AORD_AUS_zscore_60d", "HUM_Humana_ret_5d", "MSTR_Bitcoin3_ret_1d", "EWG_Germany_ret_20d", "INTC_ret_1d", "AMT_AmericanTower_ret_1d", "SJM_JM_Smucker_ret_1d", "QQQ_vol_20d", "EWM_Malaysia_zscore_60d", "CPB_CampbellSoup_zscore_60d", "T_ret_1d", "TM_Telephone_ret_1d", "Industrial_Production_zscore_60d", "EWQ_France_ret_20d", "PLD_Prologis_ret_5d", "PFE_ret_1d", "LOW_Lowes_ret_5d", "EWQ_France_zscore_60d", "EWL_Switzerland_vol_20d", "LMT_LockheedMartin_ret_1d"], "is_new": true}, {"model_id": "new_h5_GLOBAL_XGBoost_N30_t6", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 5, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "spx_abs_ret_max_5d", "vix_mean_abs_ret_5d", "EWS_Singapore_ret_5d", "EWA_Australia_zscore_60d", "EOG_EOGResources_ret_5d", "heston_var_ev_h7", "GD_GeneralDynamics_zscore_60d", "XOM_ret_20d", "EWY_Korea_ret_20d", "EWC_Canada_zscore_60d", "IWM_SmallCap_vol_20d", "AMGN_Amgen_ret_1d", "MS_MorganStanley_ret_5d", "US5Y_Rate_ret_5d", "TED_Spread_vol_20d", "HD_zscore_60d", "HangSeng_HK_ret_5d", "spx_vol_5d", "EWL_Switzerland_vol_20d", "SPY_zscore_60d", "AMT_AmericanTower_ret_1d", "EMR_Emerson_ret_20d", "ITT_ITTInc_ret_5d", "hmm_p_stress", "SJM_JM_Smucker_ret_1d", "CLX_Clorox_vol_20d", "EQIX_Equinix_ret_5d", "QQQ_vol_20d"], "is_new": true}, {"model_id": "new_h5_GLOBAL_XGBoost_N30_t7", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 5, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWY_Korea_zscore_60d", "IBEX_Spain_ret_20d", "US1Y_Rate_ret_20d", "LUV_SouthwestAir_ret_5d", "heston_var_ev_h5", "NEE_NextEra_ret_20d", "SBUX_ret_5d", "T_ret_1d", "MSTR_Bitcoin3_ret_1d", "SO_SouthernCo_ret_5d", "VVIX_ret_20d", "EOG_EOGResources_ret_5d", "MS_MorganStanley_zscore_60d", "SJM_JM_Smucker_ret_5d", "LOW_Lowes_ret_20d", "INTC_ret_1d", "CTAS_Cintas_vol_20d", "DIS_vol_20d", "Retail_Sales_zscore_60d", "BA_ret_1d", "CPB_CampbellSoup_ret_20d", "CPB_CampbellSoup_vol_20d", "TED_Spread_zscore_60d", "EWG_Germany_vol_20d", "NVDA_vol_20d", "BDX_Becton_Dickinson_ret_20d", "XOM_ret_1d", "EWM_Malaysia_ret_1d"], "is_new": true}, {"model_id": "new_h5_GLOBAL_LightGBM_N5_t0", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 5, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "Industrial_Production_zscore_60d", "HangSeng_HK_vol_20d", "LOW_Lowes_ret_20d"], "is_new": true}, {"model_id": "new_h5_GLOBAL_LightGBM_N5_t1", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 5, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "GE_ret_1d", "XOM_ret_20d", "CLX_Clorox_vol_20d"], "is_new": true}, {"model_id": "new_h5_GLOBAL_LightGBM_N5_t2", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 5, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "Brent_Oil_FRED_ret_5d", "HangSeng_HK_vol_20d", "INTC_ret_1d"], "is_new": true}, {"model_id": "new_h5_GLOBAL_LightGBM_N5_t3", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 5, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "heston_var_ev_h3", "spx_abs_ret_max_5d", "US1Y_Rate_ret_5d"], "is_new": true}, {"model_id": "new_h5_GLOBAL_LightGBM_N5_t4", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 5, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "GD_GeneralDynamics_zscore_60d", "ASX_Australia_ret_5d", "CI_Cigna_vol_20d"], "is_new": true}, {"model_id": "new_h5_GLOBAL_LightGBM_N5_t5", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 5, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "US5Y_Rate_ret_5d", "AMZN_ret_5d", "LLY_zscore_60d"], "is_new": true}, {"model_id": "new_h5_GLOBAL_LightGBM_N5_t6", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 5, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "vix_mean_abs_ret_5d", "TM_Telephone_vol_20d", "CPB_CampbellSoup_vol_20d"], "is_new": true}, {"model_id": "new_h5_GLOBAL_LightGBM_N5_t7", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 5, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "hmm_p_stress", "TED_Spread_zscore_60d", "CPB_CampbellSoup_zscore_60d"], "is_new": true}, {"model_id": "new_h5_GLOBAL_LightGBM_N8_t0", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 5, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "LOW_Lowes_ret_20d", "CPB_CampbellSoup_ret_5d", "US6M_Rate_ret_20d", "SBUX_zscore_60d", "DHR_vol_20d", "M_Macys_vol_20d"], "is_new": true}, {"model_id": "new_h5_GLOBAL_LightGBM_N8_t1", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 5, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "INTC_ret_5d", "NOC_Northrop_ret_20d", "MRK_Merck_zscore_60d", "US1Y_Rate_ret_20d", "EFFR_vol_20d", "HD_zscore_60d"], "is_new": true}, {"model_id": "new_h5_GLOBAL_LightGBM_N8_t2", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 5, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "TGT_Target_zscore_60d", "EWY_Korea_zscore_60d", "LOW_Lowes_ret_20d", "VOD_Vodafone_zscore_60d", "SJM_JM_Smucker_ret_1d", "DE_Deere_vol_20d"], "is_new": true}, {"model_id": "new_h5_GLOBAL_LightGBM_N8_t3", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 5, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "Core_PCE_zscore_60d", "CI_Cigna_vol_20d", "MSTR_Bitcoin3_ret_5d", "AMT_AmericanTower_ret_1d", "DHR_ret_1d", "Nikkei_Japan_vol_20d"], "is_new": true}, {"model_id": "new_h5_GLOBAL_LightGBM_N8_t4", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 5, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "PFE_ret_1d", "SCHW_Schwab_ret_5d", "Nikkei_Japan_zscore_60d", "DAX_Germany_vol_20d", "US7Y_Rate_ret_20d", "EQIX_Equinix_ret_5d"], "is_new": true}, {"model_id": "new_h5_GLOBAL_LightGBM_N8_t5", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 5, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWY_Korea_ret_20d", "ORCL_zscore_60d", "PAYX_Paychex_vol_20d", "JNJ_ret_1d", "LLY_zscore_60d", "LOW_Lowes_ret_5d"], "is_new": true}, {"model_id": "new_h5_GLOBAL_LightGBM_N8_t6", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 5, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "CPB_CampbellSoup_vol_20d", "VVIX_ret_20d", "EWA_Australia_ret_1d", "MSTR_Bitcoin3_ret_5d", "MSTR_Bitcoin3_ret_20d", "PAYX_Paychex_ret_20d"], "is_new": true}, {"model_id": "new_h5_GLOBAL_LightGBM_N8_t7", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 5, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "LOW_Lowes_ret_20d", "EWG_Germany_ret_20d", "IYR_US_REIT2_zscore_60d", "EQIX_Equinix_ret_5d", "DE_Deere_vol_20d", "XLF_Fin_vol_20d"], "is_new": true}, {"model_id": "new_h5_GLOBAL_LightGBM_N10_t0", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 5, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "TM_Telephone_ret_1d", "QQQ_vol_20d", "heston_var_ev_h3", "US7Y_Rate_ret_20d", "EWG_Germany_ret_20d", "spx_momentum_3d", "heston_var_ev_h5", "PLD_Prologis_ret_5d"], "is_new": true}, {"model_id": "new_h5_GLOBAL_LightGBM_N10_t1", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 5, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "Retail_Sales_zscore_60d", "LMT_LockheedMartin_vol_20d", "CPB_CampbellSoup_ret_5d", "EWL_Switzerland_vol_20d", "Brent_Oil_FRED_ret_5d", "IWM_SmallCap_vol_20d", "T10Y2Y_Spread_ret_5d", "MSTR_Bitcoin3_ret_20d"], "is_new": true}, {"model_id": "new_h5_GLOBAL_LightGBM_N10_t2", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 5, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWG_Germany_vol_20d", "TED_Spread_vol_20d", "US3Y_Rate_ret_5d", "EWY_Korea_zscore_60d", "QQQ_vol_20d", "vix_acceleration_1d", "PLD_Prologis_ret_5d", "vix_mean_abs_ret_5d"], "is_new": true}, {"model_id": "new_h5_GLOBAL_LightGBM_N10_t3", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 5, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "PAYX_Paychex_ret_20d", "US7Y_Rate_ret_20d", "EXC_Exelon_zscore_60d", "LLY_zscore_60d", "EWY_Korea_ret_20d", "XLV_Health_zscore_60d", "EWJ_Japan_vol_20d", "GILD_Gilead_ret_20d"], "is_new": true}, {"model_id": "new_h5_GLOBAL_LightGBM_N10_t4", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 5, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "NEE_NextEra_ret_20d", "HangSeng_HK_ret_1d", "EQR_Equity_ret_1d", "heston_var_ev_h7", "TXN_vol_20d", "ITT_ITTInc_ret_5d", "AXP_Amex_vol_20d", "MSTR_Bitcoin3_ret_1d"], "is_new": true}, {"model_id": "new_h5_GLOBAL_LightGBM_N10_t5", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 5, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "3M_vol_20d", "MS_MorganStanley_zscore_60d", "GD_GeneralDynamics_zscore_60d", "PFE_ret_1d", "HangSeng_HK_vol_20d", "JNJ_ret_1d", "MO_AltriaMG_ret_1d", "AMGN_Amgen_ret_1d"], "is_new": true}, {"model_id": "new_h5_GLOBAL_LightGBM_N10_t6", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 5, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "DAX_Germany_vol_20d", "NEE_NextEra_ret_20d", "3M_ret_5d", "ITT_ITTInc_ret_5d", "LUV_SouthwestAir_ret_5d", "SBUX_zscore_60d", "AXP_Amex_ret_20d", "HD_ret_1d"], "is_new": true}, {"model_id": "new_h5_GLOBAL_LightGBM_N10_t7", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 5, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWM_Malaysia_zscore_60d", "Retail_Sales_zscore_60d", "EQIX_Equinix_ret_5d", "US1Y_Rate_ret_5d", "PPL_PPL_ret_1d", "EWQ_France_ret_20d", "LOW_Lowes_ret_5d", "SBUX_zscore_60d"], "is_new": true}, {"model_id": "new_h5_GLOBAL_LightGBM_N12_t0", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 5, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "DOW_Price_zscore_60d", "XOM_ret_1d", "NFCI_ret_5d", "LLY_zscore_60d", "EWJ_Japan_vol_20d", "XLY_Disc_vol_20d", "US6M_Rate_ret_20d", "spx_abs_ret_max_5d", "VRP_ma5", "BLK_BlackRock_zscore_60d"], "is_new": true}, {"model_id": "new_h5_GLOBAL_LightGBM_N12_t1", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 5, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "CCI_CrownCastle_vol_20d", "SJM_JM_Smucker_ret_1d", "ORCL_zscore_60d", "EWY_Korea_zscore_60d", "CPB_CampbellSoup_ret_20d", "NWL_Newell_ret_20d", "TED_Spread_vol_20d", "SBUX_vol_20d", "HD_ret_20d", "HD_ret_1d"], "is_new": true}, {"model_id": "new_h5_GLOBAL_LightGBM_N12_t2", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 5, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "CPB_CampbellSoup_vol_20d", "spx_momentum_3d", "spx_abs_ret_max_5d", "Nikkei_Japan_zscore_60d", "AMD_ret_5d", "AMGN_Amgen_ret_1d", "US1Y_Rate_ret_5d", "EXC_Exelon_ret_1d", "HD_ret_20d", "EWG_Germany_vol_20d"], "is_new": true}, {"model_id": "new_h5_GLOBAL_LightGBM_N12_t3", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 5, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "3M_ret_5d", "AXP_Amex_vol_20d", "NOC_Northrop_ret_20d", "vix_mean_abs_ret_5d", "PPL_PPL_ret_1d", "ASX_Australia_ret_5d", "PAYX_Paychex_vol_20d", "XOM_ret_20d", "Nikkei_Japan_zscore_60d", "AMZN_ret_5d"], "is_new": true}, {"model_id": "new_h5_GLOBAL_LightGBM_N12_t4", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 5, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "AMGN_Amgen_ret_1d", "ASX_Australia_ret_5d", "spx_abs_ret_max_5d", "PFE_ret_1d", "heston_ev_h3", "EWA_Australia_ret_1d", "HD_ret_1d", "CLX_Clorox_vol_20d", "LOW_Lowes_ret_5d", "EFFR_ret_1d"], "is_new": true}, {"model_id": "new_h5_GLOBAL_LightGBM_N12_t5", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 5, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "DHR_vol_20d", "TM_Telephone_ret_1d", "DAX_Germany_vol_20d", "EWS_Singapore_ret_5d", "EWG_Germany_ret_20d", "BLK_BlackRock_zscore_60d", "XLF_Fin_vol_20d", "HUM_Humana_ret_5d", "EWQ_France_zscore_60d", "TED_Spread_vol_20d"], "is_new": true}, {"model_id": "new_h5_GLOBAL_LightGBM_N12_t6", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 5, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "LMT_LockheedMartin_vol_20d", "EWA_Australia_ret_1d", "PCAR_PaccarInc_ret_5d", "HangSeng_HK_ret_1d", "CPB_CampbellSoup_zscore_60d", "EOG_EOGResources_vol_20d", "NOC_Northrop_ret_20d", "AMGN_Amgen_ret_1d", "AMZN_ret_5d", "EMR_Emerson_ret_20d"], "is_new": true}, {"model_id": "new_h5_GLOBAL_LightGBM_N12_t7", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 5, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "SO_SouthernCo_ret_5d", "MSTR_Bitcoin3_ret_5d", "Brent_Oil_FRED_ret_20d", "T10Y2Y_Spread_ret_5d", "US3M_Rate_zscore_60d", "heston_ev_h3", "spx_momentum_3d", "EXC_Exelon_ret_1d", "DE_Deere_ret_5d", "AORD_AUS_zscore_60d"], "is_new": true}, {"model_id": "new_h5_GLOBAL_LightGBM_N15_t0", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 5, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "SJM_JM_Smucker_ret_5d", "DE_Deere_ret_5d", "vix_acceleration_1d", "AORD_AUS_zscore_60d", "EMR_Emerson_ret_20d", "US30Y_Rate_ret_20d", "DHR_vol_20d", "LMT_LockheedMartin_ret_1d", "EWA_Australia_zscore_60d", "Nikkei_Japan_vol_20d", "INTC_ret_5d", "DOW_Price_zscore_60d", "EFFR_ret_1d"], "is_new": true}, {"model_id": "new_h5_GLOBAL_LightGBM_N15_t1", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 5, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "TED_Spread_vol_20d", "BLK_BlackRock_zscore_60d", "EWM_Malaysia_zscore_60d", "AMD_ret_1d", "EOG_EOGResources_vol_20d", "LMT_LockheedMartin_vol_20d", "CPB_CampbellSoup_vol_20d", "LLY_zscore_60d", "PLD_Prologis_ret_5d", "Core_CPI_zscore_60d", "MS_MorganStanley_ret_5d", "BTI_BritishAmerican_ret_20d", "XLB_Materials_zscore_60d"], "is_new": true}, {"model_id": "new_h5_GLOBAL_LightGBM_N15_t2", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 5, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "DOW_Price_zscore_60d", "EWA_Australia_ret_1d", "VVIX_ret_20d", "AMD_ret_5d", "spx_momentum_3d", "HD_ret_5d", "HangSeng_HK_ret_5d", "CPB_CampbellSoup_zscore_60d", "TXN_vol_20d", "LUV_SouthwestAir_ret_5d", "ASX_Australia_vol_20d", "PAYX_Paychex_ret_20d", "BTI_BritishAmerican_ret_5d"], "is_new": true}, {"model_id": "new_h5_GLOBAL_LightGBM_N15_t3", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 5, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "BTI_BritishAmerican_ret_5d", "EXC_Exelon_zscore_60d", "ORCL_vol_20d", "EWY_Korea_ret_20d", "LOW_Lowes_ret_5d", "HD_zscore_60d", "PG_ret_20d", "M_Macys_vol_20d", "MO_AltriaMG_ret_1d", "TM_Telephone_ret_1d", "GILD_Gilead_ret_20d", "US1Y_Rate_ret_20d", "ITT_ITTInc_ret_5d"], "is_new": true}, {"model_id": "new_h5_GLOBAL_LightGBM_N15_t4", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 5, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "3M_ret_5d", "NWL_Newell_ret_20d", "US3M_Rate_zscore_60d", "EWM_Malaysia_zscore_60d", "heston_var_ev_h7", "HD_ret_20d", "SLB_Schlumberger_ret_5d", "MS_MorganStanley_ret_1d", "SBUX_vol_20d", "EWJ_Japan_vol_20d", "PCAR_PaccarInc_ret_5d", "AMGN_Amgen_ret_1d", "ES_Evergy_ret_1d"], "is_new": true}, {"model_id": "new_h5_GLOBAL_LightGBM_N15_t5", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 5, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "Retail_Sales_zscore_60d", "EXC_Exelon_zscore_60d", "EWG_Germany_ret_20d", "DHR_ret_1d", "NVDA_vol_20d", "HangSeng_HK_ret_5d", "CI_Cigna_vol_20d", "SBUX_ret_5d", "EFFR_vol_20d", "ES_Evergy_ret_1d", "LOW_Lowes_ret_5d", "Core_PCE_zscore_60d", "HD_zscore_60d"], "is_new": true}, {"model_id": "new_h5_GLOBAL_LightGBM_N15_t6", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 5, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "TM_Telephone_vol_20d", "EQR_Equity_ret_1d", "US7Y_Rate_ret_20d", "US3M_Rate_zscore_60d", "AMD_ret_5d", "XLK_Tech_zscore_60d", "CMCSA_ret_1d", "AMD_ret_1d", "CPB_CampbellSoup_zscore_60d", "BDX_Becton_Dickinson_ret_20d", "EOG_EOGResources_ret_5d", "MSTR_Bitcoin3_ret_1d", "EWS_Singapore_ret_5d"], "is_new": true}, {"model_id": "new_h5_GLOBAL_LightGBM_N15_t7", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 5, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "XLB_Materials_zscore_60d", "US7Y_Rate_ret_20d", "AORD_AUS_zscore_60d", "PAYX_Paychex_vol_20d", "EWM_Malaysia_zscore_60d", "US3M_Rate_zscore_60d", "GE_ret_1d", "US30Y_Rate_ret_20d", "US1Y_Rate_ret_5d", "EWY_Korea_ret_20d", "HangSeng_HK_vol_20d", "3M_ret_5d", "EWJ_Japan_vol_20d"], "is_new": true}, {"model_id": "new_h5_GLOBAL_LightGBM_N20_t0", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 5, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "PAYX_Paychex_zscore_60d", "spx_momentum_3d", "QQQ_vol_20d", "BTI_BritishAmerican_ret_20d", "TGT_Target_zscore_60d", "PG_ret_20d", "EQIX_Equinix_ret_5d", "VVIX_ret_20d", "SBUX_vol_20d", "AXP_Amex_vol_20d", "EWA_Australia_ret_1d", "EXC_Exelon_zscore_60d", "INTC_ret_1d", "BDX_Becton_Dickinson_ret_20d", "MRK_Merck_zscore_60d", "TXN_vol_20d", "gjr_condvar_h1", "SJM_JM_Smucker_ret_5d"], "is_new": true}, {"model_id": "new_h5_GLOBAL_LightGBM_N20_t1", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 5, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWY_Korea_zscore_60d", "M_Macys_vol_20d", "vix_mean_abs_ret_5d", "VOD_Vodafone_zscore_60d", "HD_ret_20d", "XOM_ret_20d", "EWH_HongKong_ret_5d", "XLV_Health_zscore_60d", "gjr_condvar_h1", "PPL_PPL_ret_1d", "HD_zscore_60d", "EQR_Equity_ret_1d", "DOW_Price_zscore_60d", "IWM_SmallCap_vol_20d", "SBUX_zscore_60d", "VRP_ma5", "SLB_Schlumberger_ret_5d", "EWL_Switzerland_zscore_60d"], "is_new": true}, {"model_id": "new_h5_GLOBAL_LightGBM_N20_t2", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 5, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "GILD_Gilead_ret_20d", "ITT_ITTInc_ret_5d", "EWG_Germany_vol_20d", "SLB_Schlumberger_ret_5d", "EMR_Emerson_ret_20d", "EWC_Canada_zscore_60d", "EWQ_France_zscore_60d", "EOG_EOGResources_ret_5d", "CMCSA_ret_1d", "EWL_Switzerland_vol_20d", "ASX_Australia_vol_20d", "BTI_BritishAmerican_ret_5d", "VRP_ma5", "T_ret_1d", "MSTR_Bitcoin3_ret_5d", "heston_ev_h3", "SO_SouthernCo_ret_5d", "DE_Deere_vol_20d"], "is_new": true}, {"model_id": "new_h5_GLOBAL_LightGBM_N20_t3", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 5, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "JNJ_ret_1d", "NVDA_vol_20d", "CPB_CampbellSoup_ret_5d", "BTI_BritishAmerican_ret_20d", "VVIX_ret_20d", "HD_ret_5d", "EWA_Australia_zscore_60d", "NFCI_ret_5d", "HD_ret_20d", "SBUX_zscore_60d", "IWM_SmallCap_vol_20d", "MS_MorganStanley_ret_1d", "DHR_ret_1d", "VOD_Vodafone_zscore_60d", "PAYX_Paychex_vol_20d", "US1Y_Rate_ret_5d", "TXN_vol_20d", "DOW_Price_zscore_60d"], "is_new": true}, {"model_id": "new_h5_GLOBAL_LightGBM_N20_t4", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 5, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "Brent_Oil_FRED_ret_20d", "TGT_Target_zscore_60d", "heston_var_ev_h5", "heston_ev_h3", "EXC_Exelon_ret_1d", "HD_zscore_60d", "HangSeng_HK_vol_20d", "MS_MorganStanley_ret_1d", "WTI_Oil_FRED_zscore_60d", "AVB_AvalonBay_zscore_60d", "HangSeng_HK_ret_1d", "SO_SouthernCo_ret_5d", "TED_Spread_vol_20d", "EWG_Germany_ret_20d", "IYR_US_REIT2_zscore_60d", "MS_MorganStanley_ret_5d", "HD_ret_20d", "SBUX_zscore_60d"], "is_new": true}, {"model_id": "new_h5_GLOBAL_LightGBM_N20_t5", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 5, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "PPL_PPL_ret_1d", "PFE_ret_1d", "CLX_Clorox_vol_20d", "BDX_Becton_Dickinson_ret_20d", "TXN_vol_20d", "Nikkei_Japan_zscore_60d", "VVIX_ret_20d", "MS_MorganStanley_ret_5d", "LUV_SouthwestAir_ret_5d", "EXC_Exelon_zscore_60d", "AVB_AvalonBay_zscore_60d", "MS_MorganStanley_zscore_60d", "LOW_Lowes_ret_5d", "AMZN_ret_5d", "BA_ret_1d", "DIS_vol_20d", "EWC_Canada_zscore_60d", "MO_AltriaMG_ret_1d"], "is_new": true}, {"model_id": "new_h5_GLOBAL_LightGBM_N20_t6", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 5, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "ASX_Australia_ret_5d", "PPL_PPL_ret_1d", "PCAR_PaccarInc_ret_5d", "CTAS_Cintas_vol_20d", "EWY_Korea_zscore_60d", "DHR_vol_20d", "CCI_CrownCastle_vol_20d", "AMT_AmericanTower_ret_1d", "INTC_ret_5d", "HangSeng_HK_vol_20d", "EWS_Singapore_ret_5d", "IYM_BasicMaterials_ret_20d", "CI_Cigna_vol_20d", "EWA_Australia_ret_1d", "PAYX_Paychex_ret_20d", "CPB_CampbellSoup_ret_5d", "SLB_Schlumberger_ret_1d", "EWM_Malaysia_zscore_60d"], "is_new": true}, {"model_id": "new_h5_GLOBAL_LightGBM_N20_t7", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 5, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EFFR_ret_1d", "XLK_Tech_zscore_60d", "CPB_CampbellSoup_vol_20d", "BLK_BlackRock_zscore_60d", "TXN_vol_20d", "AVB_AvalonBay_zscore_60d", "ORCL_zscore_60d", "PAYX_Paychex_vol_20d", "TM_Telephone_vol_20d", "MRK_Merck_zscore_60d", "vix_mean_abs_ret_5d", "SLB_Schlumberger_ret_1d", "ITT_ITTInc_ret_5d", "XOM_ret_20d", "WTI_Oil_FRED_zscore_60d", "CI_Cigna_vol_20d", "XLV_Health_zscore_60d", "EOG_EOGResources_vol_20d"], "is_new": true}, {"model_id": "new_h5_GLOBAL_LightGBM_N25_t0", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 5, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "ORCL_vol_20d", "VOD_Vodafone_zscore_60d", "Nikkei_Japan_vol_20d", "BDX_Becton_Dickinson_ret_20d", "EWM_Malaysia_vol_20d", "TGT_Target_zscore_60d", "EWQ_France_ret_20d", "EXC_Exelon_zscore_60d", "SBUX_zscore_60d", "TED_Spread_vol_20d", "Nikkei_Japan_zscore_60d", "DOW_Price_zscore_60d", "NEE_NextEra_ret_20d", "GILD_Gilead_ret_20d", "NWL_Newell_ret_20d", "CI_Cigna_vol_20d", "MS_MorganStanley_zscore_60d", "Core_PCE_zscore_60d", "IYM_BasicMaterials_ret_20d", "BA_ret_1d", "BTI_BritishAmerican_ret_5d", "hmm_p_stress", "NFCI_ret_5d"], "is_new": true}, {"model_id": "new_h5_GLOBAL_LightGBM_N25_t1", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 5, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "IWM_SmallCap_vol_20d", "ASX_Australia_ret_5d", "XOM_ret_1d", "AMD_ret_5d", "spx_abs_ret_max_5d", "SJM_JM_Smucker_ret_1d", "TGT_Target_zscore_60d", "Michigan_Sentiment_ret_20d", "BTI_BritishAmerican_ret_5d", "CPB_CampbellSoup_ret_20d", "EXC_Exelon_ret_1d", "AMT_AmericanTower_ret_1d", "PG_ret_20d", "Nikkei_Japan_zscore_60d", "PAYX_Paychex_zscore_60d", "XOM_ret_20d", "EWM_Malaysia_zscore_60d", "HangSeng_HK_ret_5d", "spx_vol_5d", "XLK_Tech_zscore_60d", "vix_acceleration_1d", "SLB_Schlumberger_ret_1d", "DE_Deere_ret_5d"], "is_new": true}, {"model_id": "new_h5_GLOBAL_LightGBM_N25_t2", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 5, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "NEE_NextEra_ret_20d", "HD_ret_20d", "MSTR_Bitcoin3_ret_1d", "DOW_Price_zscore_60d", "DE_Deere_vol_20d", "LLY_zscore_60d", "AXP_Amex_ret_20d", "heston_var_ev_h5", "PAYX_Paychex_vol_20d", "EXC_Exelon_zscore_60d", "PFE_ret_1d", "ORCL_vol_20d", "ENB_EnbridgeInc_ret_1d", "CMCSA_ret_1d", "US3M_Rate_vol_20d", "T_ret_1d", "US3Y_Rate_ret_5d", "EWS_Singapore_ret_5d", "TXN_vol_20d", "EWL_Switzerland_vol_20d", "INTC_ret_5d", "XLF_Fin_vol_20d", "EWM_Malaysia_vol_20d"], "is_new": true}, {"model_id": "new_h5_GLOBAL_LightGBM_N25_t3", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 5, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "US3M_Rate_vol_20d", "PAYX_Paychex_ret_20d", "BLK_BlackRock_zscore_60d", "vix_acceleration_1d", "BTI_BritishAmerican_ret_20d", "VRP_ma5", "VOD_Vodafone_zscore_60d", "DAX_Germany_zscore_60d", "LOW_Lowes_ret_5d", "SBUX_ret_5d", "IWM_SmallCap_vol_20d", "Core_PCE_zscore_60d", "CPB_CampbellSoup_vol_20d", "PFE_ret_1d", "Brent_Oil_FRED_ret_5d", "EWY_Korea_zscore_60d", "HUM_Humana_ret_5d", "AMZN_ret_5d", "EWA_Australia_zscore_60d", "HD_ret_5d", "EQIX_Equinix_ret_5d", "TM_Telephone_ret_1d", "PCAR_PaccarInc_ret_5d"], "is_new": true}, {"model_id": "new_h5_GLOBAL_LightGBM_N25_t4", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 5, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "CTAS_Cintas_vol_20d", "CPB_CampbellSoup_zscore_60d", "AXP_Amex_vol_20d", "DAX_Germany_vol_20d", "PLD_Prologis_ret_5d", "MSTR_Bitcoin3_ret_5d", "hmm_p_stress", "3M_ret_5d", "EQR_Equity_ret_1d", "ORCL_zscore_60d", "XLK_Tech_zscore_60d", "CMCSA_ret_1d", "DE_Deere_ret_5d", "NOC_Northrop_ret_20d", "Brent_Oil_FRED_ret_5d", "LLY_zscore_60d", "GILD_Gilead_ret_20d", "ASX_Australia_vol_20d", "SO_SouthernCo_ret_5d", "LUV_SouthwestAir_ret_5d", "M_Macys_vol_20d", "CI_Cigna_vol_20d", "3M_vol_20d"], "is_new": true}, {"model_id": "new_h5_GLOBAL_LightGBM_N25_t5", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 5, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "BDX_Becton_Dickinson_ret_20d", "BLK_BlackRock_zscore_60d", "EWS_Singapore_ret_5d", "EWA_Australia_ret_1d", "IBEX_Spain_ret_20d", "heston_var_ev_h3", "JNJ_ret_1d", "EWH_HongKong_ret_5d", "US3Y_Rate_ret_5d", "spx_momentum_3d", "CTAS_Cintas_vol_20d", "QQQ_vol_20d", "spx_abs_ret_max_5d", "DE_Deere_ret_5d", "DHR_vol_20d", "CMCSA_ret_1d", "Nikkei_Japan_zscore_60d", "LOW_Lowes_ret_5d", "US7Y_Rate_ret_20d", "SJM_JM_Smucker_ret_5d", "CPB_CampbellSoup_ret_20d", "CI_Cigna_vol_20d", "XLB_Materials_zscore_60d"], "is_new": true}, {"model_id": "new_h5_GLOBAL_LightGBM_N25_t6", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 5, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "MS_MorganStanley_ret_1d", "PAYX_Paychex_ret_20d", "T10Y2Y_Spread_ret_5d", "spx_abs_ret_max_5d", "WTI_Oil_FRED_zscore_60d", "JNJ_ret_1d", "EWY_Korea_ret_20d", "vix_acceleration_1d", "XLF_Fin_vol_20d", "DIS_vol_20d", "TM_Telephone_vol_20d", "BLK_BlackRock_zscore_60d", "vix_mean_abs_ret_5d", "EWQ_France_zscore_60d", "MSTR_Bitcoin3_ret_20d", "TXN_vol_20d", "DE_Deere_vol_20d", "EXC_Exelon_zscore_60d", "spx_vol_5d", "Brent_Oil_FRED_ret_5d", "SCHW_Schwab_ret_5d", "ES_Evergy_ret_1d", "VRP_ma5"], "is_new": true}, {"model_id": "new_h5_GLOBAL_LightGBM_N25_t7", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 5, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "IWM_SmallCap_vol_20d", "SO_SouthernCo_ret_5d", "EOG_EOGResources_ret_5d", "vix_mean_abs_ret_5d", "AXP_Amex_vol_20d", "TM_Telephone_ret_1d", "HD_zscore_60d", "EWJ_Japan_vol_20d", "HD_ret_1d", "ES_Evergy_ret_1d", "CMCSA_ret_1d", "Core_PCE_zscore_60d", "EFFR_vol_20d", "CPB_CampbellSoup_ret_20d", "TXN_vol_20d", "EQR_Equity_ret_1d", "hmm_p_stress", "TED_Spread_vol_20d", "spx_vol_5d", "VOD_Vodafone_zscore_60d", "XLY_Disc_vol_20d", "EXC_Exelon_zscore_60d", "LMT_LockheedMartin_ret_1d"], "is_new": true}, {"model_id": "new_h5_GLOBAL_LightGBM_N30_t0", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 5, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "HD_ret_20d", "EWY_Korea_ret_20d", "EWH_HongKong_ret_5d", "EQIX_Equinix_ret_5d", "CMCSA_ret_1d", "EFFR_ret_1d", "ITT_ITTInc_ret_5d", "VOD_Vodafone_zscore_60d", "LOW_Lowes_ret_20d", "HUM_Humana_ret_5d", "EWA_Australia_zscore_60d", "DHR_vol_20d", "GILD_Gilead_ret_20d", "HD_zscore_60d", "EWL_Switzerland_vol_20d", "XLK_Tech_zscore_60d", "EQR_Equity_ret_1d", "EOG_EOGResources_vol_20d", "BLK_BlackRock_zscore_60d", "PLD_Prologis_ret_5d", "SO_SouthernCo_ret_5d", "CTAS_Cintas_vol_20d", "AMZN_ret_5d", "IYM_BasicMaterials_ret_20d", "AORD_AUS_zscore_60d", "T_ret_1d", "US1Y_Rate_ret_5d", "GE_ret_1d"], "is_new": true}, {"model_id": "new_h5_GLOBAL_LightGBM_N30_t1", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 5, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "XOM_ret_20d", "PG_ret_20d", "EMR_Emerson_ret_20d", "EWS_Singapore_ret_5d", "EWJ_Japan_vol_20d", "M_Macys_vol_20d", "US7Y_Rate_ret_20d", "spx_momentum_3d", "MRK_Merck_zscore_60d", "CPB_CampbellSoup_ret_5d", "AXP_Amex_ret_20d", "vix_acceleration_1d", "EWL_Switzerland_vol_20d", "SO_SouthernCo_ret_5d", "heston_var_ev_h3", "BTI_BritishAmerican_ret_20d", "INTC_ret_5d", "XOM_ret_1d", "HD_ret_5d", "spx_abs_ret_max_5d", "LOW_Lowes_ret_5d", "NWL_Newell_ret_20d", "Retail_Sales_zscore_60d", "WTI_Oil_FRED_zscore_60d", "Brent_Oil_FRED_ret_20d", "LLY_zscore_60d", "US6M_Rate_ret_20d", "3M_vol_20d"], "is_new": true}, {"model_id": "new_h5_GLOBAL_LightGBM_N30_t2", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 5, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "PAYX_Paychex_ret_20d", "GILD_Gilead_ret_20d", "MO_AltriaMG_ret_1d", "INTC_ret_5d", "NEE_NextEra_ret_20d", "EWC_Canada_zscore_60d", "SBUX_zscore_60d", "US1Y_Rate_ret_5d", "ASX_Australia_vol_20d", "Core_PCE_zscore_60d", "T10Y2Y_Spread_ret_5d", "SBUX_vol_20d", "CPB_CampbellSoup_ret_20d", "IWM_SmallCap_vol_20d", "CLX_Clorox_vol_20d", "EWM_Malaysia_vol_20d", "AMD_ret_1d", "3M_vol_20d", "MS_MorganStanley_zscore_60d", "EWA_Australia_ret_1d", "HUM_Humana_ret_5d", "heston_var_ev_h3", "NVDA_vol_20d", "XOM_ret_1d", "EOG_EOGResources_vol_20d", "US3Y_Rate_ret_5d", "ENB_EnbridgeInc_ret_1d", "MSTR_Bitcoin3_ret_1d"], "is_new": true}, {"model_id": "new_h5_GLOBAL_LightGBM_N30_t3", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 5, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "TXN_vol_20d", "CTAS_Cintas_vol_20d", "LUV_SouthwestAir_ret_5d", "ORCL_zscore_60d", "VRP_ma5", "PLD_Prologis_ret_5d", "PAYX_Paychex_ret_20d", "NEE_NextEra_ret_20d", "HD_ret_5d", "DE_Deere_ret_5d", "EWG_Germany_ret_20d", "hmm_p_stress", "TED_Spread_zscore_60d", "SJM_JM_Smucker_ret_5d", "INTC_ret_1d", "ASX_Australia_vol_20d", "PAYX_Paychex_zscore_60d", "US3M_Rate_vol_20d", "AORD_AUS_zscore_60d", "HangSeng_HK_vol_20d", "PFE_ret_1d", "MRK_Merck_zscore_60d", "LMT_LockheedMartin_ret_1d", "EFFR_vol_20d", "EWC_Canada_zscore_60d", "AMD_ret_1d", "EWS_Singapore_ret_5d", "EWH_HongKong_ret_5d"], "is_new": true}, {"model_id": "new_h5_GLOBAL_LightGBM_N30_t4", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 5, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EQIX_Equinix_ret_5d", "EWM_Malaysia_vol_20d", "ENB_EnbridgeInc_ret_1d", "US3M_Rate_zscore_60d", "3M_vol_20d", "AXP_Amex_vol_20d", "CPB_CampbellSoup_vol_20d", "Core_CPI_zscore_60d", "EMR_Emerson_ret_20d", "GD_GeneralDynamics_zscore_60d", "XLV_Health_zscore_60d", "EWQ_France_ret_20d", "HD_ret_1d", "CPB_CampbellSoup_zscore_60d", "LMT_LockheedMartin_vol_20d", "SLB_Schlumberger_ret_1d", "DOW_Price_zscore_60d", "DHR_ret_1d", "CI_Cigna_vol_20d", "spx_vol_5d", "WTI_Oil_FRED_zscore_60d", "MO_AltriaMG_ret_1d", "Brent_Oil_FRED_ret_5d", "heston_var_ev_h7", "AXP_Amex_ret_20d", "US1Y_Rate_ret_20d", "TM_Telephone_vol_20d", "3M_ret_5d"], "is_new": true}, {"model_id": "new_h5_GLOBAL_LightGBM_N30_t5", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 5, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "NEE_NextEra_ret_20d", "MRK_Merck_zscore_60d", "WTI_Oil_FRED_zscore_60d", "EWM_Malaysia_ret_1d", "JNJ_ret_1d", "MS_MorganStanley_zscore_60d", "CPB_CampbellSoup_ret_20d", "GE_ret_1d", "BDX_Becton_Dickinson_ret_20d", "AMZN_ret_5d", "T_ret_1d", "CPB_CampbellSoup_ret_5d", "HangSeng_HK_ret_5d", "LMT_LockheedMartin_ret_1d", "DE_Deere_ret_5d", "US1Y_Rate_ret_20d", "MS_MorganStanley_ret_5d", "AMD_ret_1d", "Michigan_Sentiment_ret_20d", "TM_Telephone_ret_1d", "VRP_ma5", "EFFR_ret_1d", "CLX_Clorox_vol_20d", "SBUX_ret_5d", "HangSeng_HK_vol_20d", "FedFunds_zscore_60d", "SBUX_zscore_60d", "AMGN_Amgen_ret_1d"], "is_new": true}, {"model_id": "new_h5_GLOBAL_LightGBM_N30_t6", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 5, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "DHR_ret_1d", "HUM_Humana_ret_5d", "XLF_Fin_vol_20d", "XLY_Disc_vol_20d", "FedFunds_zscore_60d", "spx_vol_5d", "heston_var_ev_h7", "TXN_vol_20d", "US6M_Rate_ret_20d", "PAYX_Paychex_ret_20d", "M_Macys_vol_20d", "EWM_Malaysia_vol_20d", "US3M_Rate_zscore_60d", "Core_PCE_zscore_60d", "MSTR_Bitcoin3_ret_1d", "AORD_AUS_zscore_60d", "AXP_Amex_vol_20d", "PAYX_Paychex_vol_20d", "MS_MorganStanley_zscore_60d", "XOM_ret_1d", "HD_ret_5d", "CI_Cigna_vol_20d", "NVDA_vol_20d", "IWM_SmallCap_vol_20d", "US1Y_Rate_ret_5d", "PG_ret_20d", "AMZN_ret_5d", "Brent_Oil_FRED_ret_20d"], "is_new": true}, {"model_id": "new_h5_GLOBAL_LightGBM_N30_t7", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 5, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "NOC_Northrop_ret_20d", "EWS_Singapore_ret_5d", "Industrial_Production_zscore_60d", "HangSeng_HK_ret_1d", "CTAS_Cintas_vol_20d", "heston_ev_h3", "Nikkei_Japan_zscore_60d", "PAYX_Paychex_ret_20d", "TED_Spread_vol_20d", "gjr_condvar_h1", "IYR_US_REIT2_zscore_60d", "EWL_Switzerland_zscore_60d", "AMZN_ret_5d", "GILD_Gilead_ret_20d", "HangSeng_HK_ret_5d", "INTC_ret_1d", "MS_MorganStanley_ret_1d", "PLD_Prologis_ret_5d", "LLY_zscore_60d", "US6M_Rate_ret_20d", "SO_SouthernCo_ret_5d", "BTI_BritishAmerican_ret_20d", "US1Y_Rate_ret_5d", "Michigan_Sentiment_ret_20d", "vix_acceleration_1d", "DE_Deere_vol_20d", "spx_momentum_3d", "EWH_HongKong_ret_5d"], "is_new": true}, {"model_id": "new_h5_GLOBAL_GradientBoosting_N5_t0", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 5, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "CLX_Clorox_vol_20d", "Core_CPI_zscore_60d", "DAX_Germany_zscore_60d"], "is_new": true}, {"model_id": "new_h5_GLOBAL_GradientBoosting_N5_t1", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 5, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "DHR_vol_20d", "NVDA_vol_20d", "XOM_ret_1d"], "is_new": true}, {"model_id": "new_h5_GLOBAL_GradientBoosting_N5_t2", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 5, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "Industrial_Production_zscore_60d", "Nikkei_Japan_vol_20d", "vix_acceleration_1d"], "is_new": true}, {"model_id": "new_h5_GLOBAL_GradientBoosting_N5_t3", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 5, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "DIS_vol_20d", "heston_var_ev_h5", "BTI_BritishAmerican_ret_20d"], "is_new": true}, {"model_id": "new_h5_GLOBAL_GradientBoosting_N5_t4", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 5, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "MRK_Merck_zscore_60d", "MS_MorganStanley_ret_5d", "EWG_Germany_vol_20d"], "is_new": true}, {"model_id": "new_h5_GLOBAL_GradientBoosting_N5_t5", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 5, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "WTI_Oil_FRED_zscore_60d", "AXP_Amex_ret_20d", "EFFR_vol_20d"], "is_new": true}, {"model_id": "new_h5_GLOBAL_GradientBoosting_N5_t6", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 5, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "XOM_ret_1d", "ORCL_zscore_60d", "NFCI_ret_5d"], "is_new": true}, {"model_id": "new_h5_GLOBAL_GradientBoosting_N5_t7", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 5, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "AMT_AmericanTower_ret_1d", "DE_Deere_ret_5d", "heston_var_ev_h7"], "is_new": true}, {"model_id": "new_h5_GLOBAL_GradientBoosting_N8_t0", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 5, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EOG_EOGResources_vol_20d", "TGT_Target_zscore_60d", "VOD_Vodafone_zscore_60d", "US7Y_Rate_ret_20d", "EWM_Malaysia_zscore_60d", "NVDA_vol_20d"], "is_new": true}, {"model_id": "new_h5_GLOBAL_GradientBoosting_N8_t1", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 5, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "BTI_BritishAmerican_ret_20d", "XLK_Tech_zscore_60d", "AMT_AmericanTower_ret_1d", "TGT_Target_zscore_60d", "INTC_ret_1d", "PCAR_PaccarInc_ret_5d"], "is_new": true}, {"model_id": "new_h5_GLOBAL_GradientBoosting_N8_t2", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 5, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "DHR_vol_20d", "ES_Evergy_ret_1d", "AMGN_Amgen_ret_1d", "SO_SouthernCo_ret_5d", "heston_var_ev_h3", "MSTR_Bitcoin3_ret_1d"], "is_new": true}, {"model_id": "new_h5_GLOBAL_GradientBoosting_N8_t3", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 5, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "SJM_JM_Smucker_ret_1d", "HD_zscore_60d", "T10Y2Y_Spread_ret_5d", "MS_MorganStanley_ret_1d", "AXP_Amex_vol_20d", "EQR_Equity_ret_1d"], "is_new": true}, {"model_id": "new_h5_GLOBAL_GradientBoosting_N8_t4", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 5, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "SLB_Schlumberger_ret_5d", "CMCSA_ret_1d", "CLX_Clorox_vol_20d", "spx_vol_5d", "gjr_condvar_h1", "MS_MorganStanley_ret_1d"], "is_new": true}, {"model_id": "new_h5_GLOBAL_GradientBoosting_N8_t5", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 5, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "PAYX_Paychex_ret_20d", "EWS_Singapore_ret_5d", "EOG_EOGResources_vol_20d", "spx_abs_ret_max_5d", "SJM_JM_Smucker_ret_5d", "MO_AltriaMG_ret_1d"], "is_new": true}, {"model_id": "new_h5_GLOBAL_GradientBoosting_N8_t6", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 5, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "GD_GeneralDynamics_zscore_60d", "US1Y_Rate_ret_5d", "EXC_Exelon_ret_1d", "3M_ret_5d", "T10Y2Y_Spread_ret_5d", "LOW_Lowes_ret_5d"], "is_new": true}, {"model_id": "new_h5_GLOBAL_GradientBoosting_N8_t7", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 5, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "WTI_Oil_FRED_zscore_60d", "TED_Spread_zscore_60d", "DAX_Germany_zscore_60d", "ORCL_zscore_60d", "DIS_vol_20d", "NEE_NextEra_ret_20d"], "is_new": true}, {"model_id": "new_h5_GLOBAL_GradientBoosting_N10_t0", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 5, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "ORCL_vol_20d", "SLB_Schlumberger_ret_5d", "EFFR_vol_20d", "XLY_Disc_vol_20d", "WTI_Oil_FRED_zscore_60d", "PPL_PPL_ret_1d", "DIS_vol_20d", "T10Y2Y_Spread_ret_5d"], "is_new": true}, {"model_id": "new_h5_GLOBAL_GradientBoosting_N10_t1", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 5, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "SJM_JM_Smucker_ret_5d", "Michigan_Sentiment_ret_20d", "SLB_Schlumberger_ret_1d", "AXP_Amex_ret_20d", "FedFunds_zscore_60d", "EOG_EOGResources_ret_5d", "TED_Spread_vol_20d", "TM_Telephone_ret_1d"], "is_new": true}, {"model_id": "new_h5_GLOBAL_GradientBoosting_N10_t2", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 5, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "ORCL_zscore_60d", "EFFR_ret_1d", "HUM_Humana_ret_5d", "DAX_Germany_vol_20d", "XLF_Fin_vol_20d", "LOW_Lowes_ret_20d", "HD_ret_5d", "XOM_ret_20d"], "is_new": true}, {"model_id": "new_h5_GLOBAL_GradientBoosting_N10_t3", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 5, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "ORCL_vol_20d", "EOG_EOGResources_ret_5d", "Michigan_Sentiment_ret_20d", "US3M_Rate_zscore_60d", "MSTR_Bitcoin3_ret_20d", "PG_ret_20d", "EOG_EOGResources_vol_20d", "TGT_Target_zscore_60d"], "is_new": true}, {"model_id": "new_h5_GLOBAL_GradientBoosting_N10_t4", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 5, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "Nikkei_Japan_vol_20d", "LMT_LockheedMartin_ret_1d", "IYR_US_REIT2_zscore_60d", "Industrial_Production_zscore_60d", "BTI_BritishAmerican_ret_20d", "FedFunds_zscore_60d", "Nikkei_Japan_zscore_60d", "LOW_Lowes_ret_5d"], "is_new": true}, {"model_id": "new_h5_GLOBAL_GradientBoosting_N10_t5", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 5, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWG_Germany_ret_20d", "BDX_Becton_Dickinson_ret_20d", "VVIX_ret_20d", "LLY_zscore_60d", "EWM_Malaysia_vol_20d", "EFFR_vol_20d", "CPB_CampbellSoup_vol_20d", "TM_Telephone_vol_20d"], "is_new": true}, {"model_id": "new_h5_GLOBAL_GradientBoosting_N10_t6", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 5, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWY_Korea_ret_20d", "AXP_Amex_ret_20d", "IWM_SmallCap_vol_20d", "vix_acceleration_1d", "SLB_Schlumberger_ret_5d", "MSTR_Bitcoin3_ret_20d", "MO_AltriaMG_ret_1d", "FedFunds_zscore_60d"], "is_new": true}, {"model_id": "new_h5_GLOBAL_GradientBoosting_N10_t7", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 5, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "NWL_Newell_ret_20d", "SJM_JM_Smucker_ret_1d", "CPB_CampbellSoup_zscore_60d", "XLB_Materials_zscore_60d", "CPB_CampbellSoup_vol_20d", "SCHW_Schwab_ret_5d", "HD_ret_20d", "EOG_EOGResources_vol_20d"], "is_new": true}, {"model_id": "new_h5_GLOBAL_GradientBoosting_N12_t0", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 5, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "IYM_BasicMaterials_ret_20d", "vix_acceleration_1d", "NFCI_ret_5d", "MSTR_Bitcoin3_ret_1d", "vix_mean_abs_ret_5d", "QQQ_vol_20d", "CPB_CampbellSoup_ret_5d", "PAYX_Paychex_zscore_60d", "spx_abs_ret_max_5d", "LUV_SouthwestAir_ret_5d"], "is_new": true}, {"model_id": "new_h5_GLOBAL_GradientBoosting_N12_t1", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 5, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "XLV_Health_zscore_60d", "PCAR_PaccarInc_ret_5d", "LMT_LockheedMartin_ret_1d", "SPY_zscore_60d", "MSTR_Bitcoin3_ret_1d", "CPB_CampbellSoup_zscore_60d", "EWY_Korea_ret_20d", "spx_abs_ret_max_5d", "US3M_Rate_vol_20d", "EWM_Malaysia_vol_20d"], "is_new": true}, {"model_id": "new_h5_GLOBAL_GradientBoosting_N12_t2", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 5, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "AVB_AvalonBay_zscore_60d", "AXP_Amex_vol_20d", "HangSeng_HK_ret_1d", "HangSeng_HK_vol_20d", "BTI_BritishAmerican_ret_20d", "TM_Telephone_vol_20d", "LOW_Lowes_ret_20d", "gjr_condvar_h1", "BDX_Becton_Dickinson_ret_20d", "AMD_ret_1d"], "is_new": true}, {"model_id": "new_h5_GLOBAL_GradientBoosting_N12_t3", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 5, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "AMD_ret_1d", "SJM_JM_Smucker_ret_1d", "spx_abs_ret_max_5d", "VVIX_ret_20d", "SPY_zscore_60d", "HangSeng_HK_vol_20d", "EWQ_France_zscore_60d", "MSTR_Bitcoin3_ret_1d", "EWM_Malaysia_zscore_60d", "GILD_Gilead_ret_20d"], "is_new": true}, {"model_id": "new_h5_GLOBAL_GradientBoosting_N12_t4", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 5, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "Nikkei_Japan_vol_20d", "INTC_ret_5d", "TXN_vol_20d", "SLB_Schlumberger_ret_5d", "QQQ_vol_20d", "XLB_Materials_zscore_60d", "EWA_Australia_zscore_60d", "HD_zscore_60d", "AORD_AUS_zscore_60d", "XOM_ret_20d"], "is_new": true}, {"model_id": "new_h5_GLOBAL_GradientBoosting_N12_t5", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 5, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "Brent_Oil_FRED_ret_20d", "EWG_Germany_ret_20d", "QQQ_vol_20d", "AMGN_Amgen_ret_1d", "PAYX_Paychex_ret_20d", "T10Y2Y_Spread_ret_5d", "PPL_PPL_ret_1d", "AORD_AUS_zscore_60d", "EWM_Malaysia_vol_20d", "spx_momentum_3d"], "is_new": true}, {"model_id": "new_h5_GLOBAL_GradientBoosting_N12_t6", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 5, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "BTI_BritishAmerican_ret_20d", "CTAS_Cintas_vol_20d", "SLB_Schlumberger_ret_1d", "DOW_Price_zscore_60d", "MS_MorganStanley_ret_1d", "gjr_condvar_h1", "BDX_Becton_Dickinson_ret_20d", "Nikkei_Japan_zscore_60d", "AXP_Amex_ret_20d", "TED_Spread_zscore_60d"], "is_new": true}, {"model_id": "new_h5_GLOBAL_GradientBoosting_N12_t7", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 5, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "NVDA_vol_20d", "GE_ret_1d", "heston_var_ev_h5", "M_Macys_vol_20d", "SJM_JM_Smucker_ret_1d", "AMT_AmericanTower_ret_1d", "JNJ_ret_1d", "HangSeng_HK_ret_1d", "EOG_EOGResources_ret_5d", "3M_ret_5d"], "is_new": true}, {"model_id": "new_h5_GLOBAL_GradientBoosting_N15_t0", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 5, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "AMD_ret_1d", "ASX_Australia_vol_20d", "BTI_BritishAmerican_ret_20d", "TED_Spread_zscore_60d", "spx_abs_ret_max_5d", "DOW_Price_zscore_60d", "NVDA_vol_20d", "3M_vol_20d", "M_Macys_vol_20d", "ES_Evergy_ret_1d", "BLK_BlackRock_zscore_60d", "Core_PCE_zscore_60d", "DAX_Germany_vol_20d"], "is_new": true}, {"model_id": "new_h5_GLOBAL_GradientBoosting_N15_t1", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 5, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "3M_vol_20d", "XLF_Fin_vol_20d", "AVB_AvalonBay_zscore_60d", "US7Y_Rate_ret_20d", "BA_ret_1d", "HangSeng_HK_vol_20d", "EMR_Emerson_ret_20d", "EWS_Singapore_ret_5d", "DOW_Price_zscore_60d", "NEE_NextEra_ret_20d", "heston_var_ev_h3", "NFCI_ret_5d", "EWM_Malaysia_ret_1d"], "is_new": true}, {"model_id": "new_h5_GLOBAL_GradientBoosting_N15_t2", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 5, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "IBEX_Spain_ret_20d", "Industrial_Production_zscore_60d", "ASX_Australia_vol_20d", "heston_ev_h3", "XOM_ret_20d", "FedFunds_zscore_60d", "ASX_Australia_ret_5d", "Retail_Sales_zscore_60d", "VRP_ma5", "HUM_Humana_ret_5d", "DIS_vol_20d", "heston_var_ev_h5", "LMT_LockheedMartin_ret_1d"], "is_new": true}, {"model_id": "new_h5_GLOBAL_GradientBoosting_N15_t3", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 5, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "ENB_EnbridgeInc_ret_1d", "BTI_BritishAmerican_ret_20d", "EWM_Malaysia_vol_20d", "XOM_ret_1d", "heston_var_ev_h5", "IWM_SmallCap_vol_20d", "DE_Deere_ret_5d", "AMZN_ret_5d", "PPL_PPL_ret_1d", "NFCI_ret_5d", "FedFunds_zscore_60d", "EWY_Korea_ret_20d", "JNJ_ret_1d"], "is_new": true}, {"model_id": "new_h5_GLOBAL_GradientBoosting_N15_t4", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 5, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "NVDA_vol_20d", "DAX_Germany_vol_20d", "heston_var_ev_h5", "SLB_Schlumberger_ret_5d", "EWG_Germany_ret_20d", "PPL_PPL_ret_1d", "AVB_AvalonBay_zscore_60d", "JNJ_ret_1d", "PCAR_PaccarInc_ret_5d", "Retail_Sales_zscore_60d", "HangSeng_HK_vol_20d", "IBEX_Spain_ret_20d", "CI_Cigna_vol_20d"], "is_new": true}, {"model_id": "new_h5_GLOBAL_GradientBoosting_N15_t5", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 5, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "Core_CPI_zscore_60d", "HD_ret_5d", "IWM_SmallCap_vol_20d", "US3M_Rate_vol_20d", "SO_SouthernCo_ret_5d", "HangSeng_HK_vol_20d", "spx_abs_ret_max_5d", "SPY_zscore_60d", "XOM_ret_20d", "GD_GeneralDynamics_zscore_60d", "Retail_Sales_zscore_60d", "heston_var_ev_h7", "CCI_CrownCastle_vol_20d"], "is_new": true}, {"model_id": "new_h5_GLOBAL_GradientBoosting_N15_t6", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 5, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "SCHW_Schwab_ret_5d", "ITT_ITTInc_ret_5d", "MSTR_Bitcoin3_ret_1d", "US1Y_Rate_ret_20d", "EOG_EOGResources_ret_5d", "heston_ev_h3", "M_Macys_vol_20d", "TGT_Target_zscore_60d", "BTI_BritishAmerican_ret_20d", "EXC_Exelon_zscore_60d", "vix_mean_abs_ret_5d", "Core_CPI_zscore_60d", "LUV_SouthwestAir_ret_5d"], "is_new": true}, {"model_id": "new_h5_GLOBAL_GradientBoosting_N15_t7", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 5, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "IYR_US_REIT2_zscore_60d", "PCAR_PaccarInc_ret_5d", "AMD_ret_5d", "HD_ret_20d", "AMGN_Amgen_ret_1d", "CPB_CampbellSoup_ret_20d", "SBUX_vol_20d", "EWA_Australia_ret_1d", "heston_var_ev_h5", "EWM_Malaysia_vol_20d", "PFE_ret_1d", "US7Y_Rate_ret_20d", "CMCSA_ret_1d"], "is_new": true}, {"model_id": "new_h5_GLOBAL_GradientBoosting_N20_t0", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 5, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWM_Malaysia_vol_20d", "Core_PCE_zscore_60d", "T10Y2Y_Spread_ret_5d", "SPY_zscore_60d", "EWH_HongKong_ret_5d", "XLK_Tech_zscore_60d", "Industrial_Production_zscore_60d", "IYM_BasicMaterials_ret_20d", "PAYX_Paychex_vol_20d", "ASX_Australia_vol_20d", "CMCSA_ret_1d", "NVDA_vol_20d", "AORD_AUS_zscore_60d", "CPB_CampbellSoup_ret_20d", "Nikkei_Japan_vol_20d", "CLX_Clorox_vol_20d", "spx_vol_5d", "XLY_Disc_vol_20d"], "is_new": true}, {"model_id": "new_h5_GLOBAL_GradientBoosting_N20_t1", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 5, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "LLY_zscore_60d", "EWY_Korea_ret_20d", "IWM_SmallCap_vol_20d", "DHR_ret_1d", "LOW_Lowes_ret_5d", "EWQ_France_ret_20d", "PAYX_Paychex_zscore_60d", "AMD_ret_5d", "XLK_Tech_zscore_60d", "HangSeng_HK_ret_1d", "EFFR_ret_1d", "AORD_AUS_zscore_60d", "MS_MorganStanley_ret_1d", "LOW_Lowes_ret_20d", "XOM_ret_1d", "US7Y_Rate_ret_20d", "VOD_Vodafone_zscore_60d", "ITT_ITTInc_ret_5d"], "is_new": true}, {"model_id": "new_h5_GLOBAL_GradientBoosting_N20_t2", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 5, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "T_ret_1d", "AMD_ret_5d", "US7Y_Rate_ret_20d", "US6M_Rate_ret_20d", "INTC_ret_5d", "US30Y_Rate_ret_20d", "XLF_Fin_vol_20d", "EFFR_ret_1d", "spx_abs_ret_max_5d", "spx_vol_5d", "SPY_zscore_60d", "DHR_ret_1d", "Core_CPI_zscore_60d", "heston_var_ev_h7", "IYR_US_REIT2_zscore_60d", "HangSeng_HK_vol_20d", "heston_var_ev_h5", "Nikkei_Japan_vol_20d"], "is_new": true}, {"model_id": "new_h5_GLOBAL_GradientBoosting_N20_t3", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 5, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "Retail_Sales_zscore_60d", "DHR_ret_1d", "spx_momentum_3d", "Industrial_Production_zscore_60d", "Core_CPI_zscore_60d", "TM_Telephone_vol_20d", "SJM_JM_Smucker_ret_5d", "EWM_Malaysia_vol_20d", "EWA_Australia_zscore_60d", "US30Y_Rate_ret_20d", "gjr_condvar_h1", "VRP_ma5", "IYR_US_REIT2_zscore_60d", "EWS_Singapore_ret_5d", "HangSeng_HK_ret_5d", "VOD_Vodafone_zscore_60d", "CCI_CrownCastle_vol_20d", "ASX_Australia_ret_5d"], "is_new": true}, {"model_id": "new_h5_GLOBAL_GradientBoosting_N20_t4", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 5, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "HD_ret_5d", "VRP_ma5", "EWL_Switzerland_zscore_60d", "MO_AltriaMG_ret_1d", "HUM_Humana_ret_5d", "EXC_Exelon_zscore_60d", "XLY_Disc_vol_20d", "heston_var_ev_h7", "SBUX_vol_20d", "AMD_ret_5d", "PAYX_Paychex_vol_20d", "Industrial_Production_zscore_60d", "INTC_ret_5d", "HD_zscore_60d", "LOW_Lowes_ret_5d", "EWA_Australia_ret_1d", "TGT_Target_zscore_60d", "TXN_vol_20d"], "is_new": true}, {"model_id": "new_h5_GLOBAL_GradientBoosting_N20_t5", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 5, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "AORD_AUS_zscore_60d", "US3M_Rate_vol_20d", "Core_PCE_zscore_60d", "PAYX_Paychex_vol_20d", "ASX_Australia_ret_5d", "SJM_JM_Smucker_ret_5d", "ITT_ITTInc_ret_5d", "SBUX_vol_20d", "HangSeng_HK_vol_20d", "EWJ_Japan_vol_20d", "TED_Spread_vol_20d", "DHR_vol_20d", "Nikkei_Japan_vol_20d", "SJM_JM_Smucker_ret_1d", "TED_Spread_zscore_60d", "DOW_Price_zscore_60d", "US30Y_Rate_ret_20d", "EFFR_vol_20d"], "is_new": true}, {"model_id": "new_h5_GLOBAL_GradientBoosting_N20_t6", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 5, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "TED_Spread_vol_20d", "AXP_Amex_ret_20d", "PLD_Prologis_ret_5d", "US5Y_Rate_ret_5d", "US1Y_Rate_ret_20d", "BTI_BritishAmerican_ret_5d", "LOW_Lowes_ret_20d", "DAX_Germany_vol_20d", "EWM_Malaysia_zscore_60d", "CPB_CampbellSoup_zscore_60d", "HangSeng_HK_vol_20d", "ORCL_vol_20d", "US7Y_Rate_ret_20d", "EFFR_vol_20d", "US6M_Rate_ret_20d", "XOM_ret_1d", "EWL_Switzerland_zscore_60d", "EWG_Germany_vol_20d"], "is_new": true}, {"model_id": "new_h5_GLOBAL_GradientBoosting_N20_t7", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 5, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "LOW_Lowes_ret_20d", "SLB_Schlumberger_ret_1d", "SBUX_vol_20d", "EOG_EOGResources_ret_5d", "EWM_Malaysia_ret_1d", "MSTR_Bitcoin3_ret_5d", "EWY_Korea_ret_20d", "spx_abs_ret_max_5d", "MSTR_Bitcoin3_ret_1d", "DOW_Price_zscore_60d", "TED_Spread_vol_20d", "IYM_BasicMaterials_ret_20d", "SJM_JM_Smucker_ret_5d", "HD_ret_5d", "heston_var_ev_h7", "VOD_Vodafone_zscore_60d", "TXN_vol_20d", "Brent_Oil_FRED_ret_20d"], "is_new": true}, {"model_id": "new_h5_GLOBAL_GradientBoosting_N25_t0", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 5, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "LOW_Lowes_ret_5d", "TED_Spread_zscore_60d", "PFE_ret_1d", "EWM_Malaysia_ret_1d", "TGT_Target_zscore_60d", "NEE_NextEra_ret_20d", "TXN_vol_20d", "TED_Spread_vol_20d", "AORD_AUS_zscore_60d", "LUV_SouthwestAir_ret_5d", "AXP_Amex_vol_20d", "spx_vol_5d", "HangSeng_HK_ret_5d", "INTC_ret_1d", "CPB_CampbellSoup_ret_20d", "DAX_Germany_vol_20d", "IYR_US_REIT2_zscore_60d", "EQIX_Equinix_ret_5d", "HUM_Humana_ret_5d", "PG_ret_20d", "Brent_Oil_FRED_ret_5d", "heston_var_ev_h5", "ORCL_zscore_60d"], "is_new": true}, {"model_id": "new_h5_GLOBAL_GradientBoosting_N25_t1", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 5, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "AMZN_ret_5d", "EWC_Canada_zscore_60d", "3M_ret_5d", "BA_ret_1d", "EWS_Singapore_ret_5d", "JNJ_ret_1d", "DE_Deere_vol_20d", "QQQ_vol_20d", "IWM_SmallCap_vol_20d", "CPB_CampbellSoup_ret_5d", "PG_ret_20d", "EOG_EOGResources_vol_20d", "TED_Spread_zscore_60d", "MS_MorganStanley_ret_5d", "EWL_Switzerland_zscore_60d", "DAX_Germany_zscore_60d", "HD_ret_1d", "HangSeng_HK_vol_20d", "T_ret_1d", "SCHW_Schwab_ret_5d", "SLB_Schlumberger_ret_1d", "EWQ_France_zscore_60d", "NWL_Newell_ret_20d"], "is_new": true}, {"model_id": "new_h5_GLOBAL_GradientBoosting_N25_t2", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 5, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "heston_var_ev_h5", "MO_AltriaMG_ret_1d", "heston_var_ev_h7", "LOW_Lowes_ret_20d", "TXN_vol_20d", "ES_Evergy_ret_1d", "SLB_Schlumberger_ret_1d", "Brent_Oil_FRED_ret_20d", "HUM_Humana_ret_5d", "Michigan_Sentiment_ret_20d", "gjr_condvar_h1", "CPB_CampbellSoup_ret_5d", "PLD_Prologis_ret_5d", "ENB_EnbridgeInc_ret_1d", "PPL_PPL_ret_1d", "AORD_AUS_zscore_60d", "CI_Cigna_vol_20d", "US1Y_Rate_ret_5d", "US3M_Rate_zscore_60d", "Core_CPI_zscore_60d", "EWH_HongKong_ret_5d", "SO_SouthernCo_ret_5d", "XLB_Materials_zscore_60d"], "is_new": true}, {"model_id": "new_h5_GLOBAL_GradientBoosting_N25_t3", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 5, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "DE_Deere_ret_5d", "TM_Telephone_vol_20d", "EWS_Singapore_ret_5d", "Michigan_Sentiment_ret_20d", "3M_ret_5d", "ORCL_vol_20d", "XLF_Fin_vol_20d", "MSTR_Bitcoin3_ret_20d", "NWL_Newell_ret_20d", "TM_Telephone_ret_1d", "SLB_Schlumberger_ret_1d", "GE_ret_1d", "EWY_Korea_zscore_60d", "TXN_vol_20d", "SCHW_Schwab_ret_5d", "US7Y_Rate_ret_20d", "HD_ret_5d", "spx_vol_5d", "CLX_Clorox_vol_20d", "PAYX_Paychex_zscore_60d", "EWA_Australia_zscore_60d", "EWC_Canada_zscore_60d", "DE_Deere_vol_20d"], "is_new": true}, {"model_id": "new_h5_GLOBAL_GradientBoosting_N25_t4", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 5, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "HD_ret_1d", "US1Y_Rate_ret_20d", "AORD_AUS_zscore_60d", "PG_ret_20d", "EMR_Emerson_ret_20d", "PCAR_PaccarInc_ret_5d", "HD_ret_20d", "EWL_Switzerland_vol_20d", "IYR_US_REIT2_zscore_60d", "SLB_Schlumberger_ret_5d", "LUV_SouthwestAir_ret_5d", "MSTR_Bitcoin3_ret_5d", "US3Y_Rate_ret_5d", "EWJ_Japan_vol_20d", "DAX_Germany_zscore_60d", "SJM_JM_Smucker_ret_5d", "SBUX_zscore_60d", "FedFunds_zscore_60d", "EWS_Singapore_ret_5d", "HD_zscore_60d", "DHR_ret_1d", "BA_ret_1d", "IYM_BasicMaterials_ret_20d"], "is_new": true}, {"model_id": "new_h5_GLOBAL_GradientBoosting_N25_t5", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 5, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "US7Y_Rate_ret_20d", "3M_vol_20d", "LMT_LockheedMartin_vol_20d", "AVB_AvalonBay_zscore_60d", "US30Y_Rate_ret_20d", "gjr_condvar_h1", "EWH_HongKong_ret_5d", "HD_ret_20d", "LMT_LockheedMartin_ret_1d", "BDX_Becton_Dickinson_ret_20d", "EWC_Canada_zscore_60d", "PAYX_Paychex_zscore_60d", "EWM_Malaysia_zscore_60d", "MRK_Merck_zscore_60d", "AMD_ret_5d", "XLF_Fin_vol_20d", "EQIX_Equinix_ret_5d", "MSTR_Bitcoin3_ret_5d", "US3M_Rate_zscore_60d", "EOG_EOGResources_vol_20d", "JNJ_ret_1d", "EMR_Emerson_ret_20d", "Nikkei_Japan_vol_20d"], "is_new": true}, {"model_id": "new_h5_GLOBAL_GradientBoosting_N25_t6", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 5, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "LUV_SouthwestAir_ret_5d", "EXC_Exelon_ret_1d", "CLX_Clorox_vol_20d", "T10Y2Y_Spread_ret_5d", "TXN_vol_20d", "hmm_p_stress", "DE_Deere_vol_20d", "EQIX_Equinix_ret_5d", "SLB_Schlumberger_ret_1d", "PAYX_Paychex_zscore_60d", "ASX_Australia_ret_5d", "BDX_Becton_Dickinson_ret_20d", "DHR_vol_20d", "BTI_BritishAmerican_ret_5d", "EWL_Switzerland_vol_20d", "Nikkei_Japan_vol_20d", "spx_vol_5d", "XLV_Health_zscore_60d", "US6M_Rate_ret_20d", "US3M_Rate_zscore_60d", "EOG_EOGResources_ret_5d", "EWH_HongKong_ret_5d", "EFFR_ret_1d"], "is_new": true}, {"model_id": "new_h5_GLOBAL_GradientBoosting_N25_t7", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 5, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "IYM_BasicMaterials_ret_20d", "CPB_CampbellSoup_ret_20d", "EMR_Emerson_ret_20d", "DIS_vol_20d", "PLD_Prologis_ret_5d", "NVDA_vol_20d", "3M_ret_5d", "PAYX_Paychex_zscore_60d", "TGT_Target_zscore_60d", "hmm_p_stress", "Nikkei_Japan_vol_20d", "3M_vol_20d", "PAYX_Paychex_vol_20d", "Michigan_Sentiment_ret_20d", "HD_zscore_60d", "HD_ret_5d", "DAX_Germany_zscore_60d", "heston_var_ev_h7", "PCAR_PaccarInc_ret_5d", "ASX_Australia_vol_20d", "SCHW_Schwab_ret_5d", "SBUX_zscore_60d", "CPB_CampbellSoup_vol_20d"], "is_new": true}, {"model_id": "new_h5_GLOBAL_GradientBoosting_N30_t0", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 5, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWY_Korea_ret_20d", "CLX_Clorox_vol_20d", "EOG_EOGResources_vol_20d", "T_ret_1d", "XLY_Disc_vol_20d", "US7Y_Rate_ret_20d", "SJM_JM_Smucker_ret_5d", "INTC_ret_1d", "Core_CPI_zscore_60d", "vix_acceleration_1d", "SLB_Schlumberger_ret_5d", "Nikkei_Japan_zscore_60d", "3M_ret_5d", "LLY_zscore_60d", "GD_GeneralDynamics_zscore_60d", "DAX_Germany_zscore_60d", "EWG_Germany_ret_20d", "Brent_Oil_FRED_ret_5d", "CPB_CampbellSoup_ret_5d", "ENB_EnbridgeInc_ret_1d", "MSTR_Bitcoin3_ret_1d", "DHR_ret_1d", "LMT_LockheedMartin_vol_20d", "PAYX_Paychex_zscore_60d", "GE_ret_1d", "SO_SouthernCo_ret_5d", "HD_ret_1d", "MS_MorganStanley_zscore_60d"], "is_new": true}, {"model_id": "new_h5_GLOBAL_GradientBoosting_N30_t1", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 5, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "IWM_SmallCap_vol_20d", "JNJ_ret_1d", "AMZN_ret_5d", "LOW_Lowes_ret_5d", "PAYX_Paychex_zscore_60d", "INTC_ret_1d", "BTI_BritishAmerican_ret_20d", "US3M_Rate_zscore_60d", "GD_GeneralDynamics_zscore_60d", "EWM_Malaysia_zscore_60d", "EXC_Exelon_ret_1d", "DAX_Germany_zscore_60d", "Industrial_Production_zscore_60d", "GILD_Gilead_ret_20d", "SJM_JM_Smucker_ret_5d", "DHR_vol_20d", "INTC_ret_5d", "AMD_ret_1d", "TM_Telephone_ret_1d", "LUV_SouthwestAir_ret_5d", "heston_var_ev_h7", "CCI_CrownCastle_vol_20d", "spx_momentum_3d", "HangSeng_HK_vol_20d", "US3Y_Rate_ret_5d", "FedFunds_zscore_60d", "spx_abs_ret_max_5d", "MRK_Merck_zscore_60d"], "is_new": true}, {"model_id": "new_h5_GLOBAL_GradientBoosting_N30_t2", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 5, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "MO_AltriaMG_ret_1d", "EWL_Switzerland_vol_20d", "EXC_Exelon_ret_1d", "EOG_EOGResources_ret_5d", "JNJ_ret_1d", "HangSeng_HK_ret_1d", "EFFR_ret_1d", "DOW_Price_zscore_60d", "SJM_JM_Smucker_ret_5d", "US30Y_Rate_ret_20d", "AMT_AmericanTower_ret_1d", "EWM_Malaysia_vol_20d", "heston_var_ev_h7", "NVDA_vol_20d", "3M_ret_5d", "spx_momentum_3d", "US3M_Rate_zscore_60d", "Retail_Sales_zscore_60d", "HD_zscore_60d", "heston_ev_h3", "AMZN_ret_5d", "ITT_ITTInc_ret_5d", "PCAR_PaccarInc_ret_5d", "NWL_Newell_ret_20d", "US1Y_Rate_ret_20d", "PAYX_Paychex_zscore_60d", "EWC_Canada_zscore_60d", "EWH_HongKong_ret_5d"], "is_new": true}, {"model_id": "new_h5_GLOBAL_GradientBoosting_N30_t3", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 5, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "T10Y2Y_Spread_ret_5d", "T_ret_1d", "XLB_Materials_zscore_60d", "PCAR_PaccarInc_ret_5d", "MS_MorganStanley_zscore_60d", "CMCSA_ret_1d", "CI_Cigna_vol_20d", "BA_ret_1d", "WTI_Oil_FRED_zscore_60d", "PAYX_Paychex_vol_20d", "Nikkei_Japan_zscore_60d", "EWG_Germany_ret_20d", "ASX_Australia_vol_20d", "EMR_Emerson_ret_20d", "SPY_zscore_60d", "NWL_Newell_ret_20d", "XLK_Tech_zscore_60d", "LLY_zscore_60d", "EWS_Singapore_ret_5d", "LUV_SouthwestAir_ret_5d", "US1Y_Rate_ret_5d", "Brent_Oil_FRED_ret_5d", "vix_acceleration_1d", "US6M_Rate_ret_20d", "US7Y_Rate_ret_20d", "LMT_LockheedMartin_vol_20d", "PG_ret_20d", "CPB_CampbellSoup_zscore_60d"], "is_new": true}, {"model_id": "new_h5_GLOBAL_GradientBoosting_N30_t4", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 5, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "FedFunds_zscore_60d", "T10Y2Y_Spread_ret_5d", "M_Macys_vol_20d", "US1Y_Rate_ret_5d", "TM_Telephone_ret_1d", "ENB_EnbridgeInc_ret_1d", "heston_ev_h3", "EWQ_France_ret_20d", "spx_momentum_3d", "LOW_Lowes_ret_20d", "LLY_zscore_60d", "Core_PCE_zscore_60d", "gjr_condvar_h1", "CPB_CampbellSoup_vol_20d", "AORD_AUS_zscore_60d", "US7Y_Rate_ret_20d", "EWS_Singapore_ret_5d", "EWY_Korea_zscore_60d", "Michigan_Sentiment_ret_20d", "NWL_Newell_ret_20d", "ORCL_zscore_60d", "heston_var_ev_h5", "AXP_Amex_vol_20d", "SLB_Schlumberger_ret_1d", "TM_Telephone_vol_20d", "3M_vol_20d", "EXC_Exelon_zscore_60d", "US5Y_Rate_ret_5d"], "is_new": true}, {"model_id": "new_h5_GLOBAL_GradientBoosting_N30_t5", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 5, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "SBUX_vol_20d", "DAX_Germany_zscore_60d", "SCHW_Schwab_ret_5d", "HD_ret_1d", "Nikkei_Japan_zscore_60d", "AMD_ret_5d", "AVB_AvalonBay_zscore_60d", "CCI_CrownCastle_vol_20d", "DAX_Germany_vol_20d", "CLX_Clorox_vol_20d", "LUV_SouthwestAir_ret_5d", "ORCL_vol_20d", "NOC_Northrop_ret_20d", "EWQ_France_ret_20d", "Core_PCE_zscore_60d", "BTI_BritishAmerican_ret_20d", "AMZN_ret_5d", "SLB_Schlumberger_ret_5d", "GD_GeneralDynamics_zscore_60d", "TXN_vol_20d", "MO_AltriaMG_ret_1d", "JNJ_ret_1d", "PCAR_PaccarInc_ret_5d", "EWA_Australia_zscore_60d", "EWM_Malaysia_ret_1d", "SPY_zscore_60d", "CPB_CampbellSoup_ret_5d", "DIS_vol_20d"], "is_new": true}, {"model_id": "new_h5_GLOBAL_GradientBoosting_N30_t6", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 5, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "SBUX_zscore_60d", "Michigan_Sentiment_ret_20d", "MSTR_Bitcoin3_ret_5d", "US7Y_Rate_ret_20d", "MS_MorganStanley_zscore_60d", "GILD_Gilead_ret_20d", "T_ret_1d", "FedFunds_zscore_60d", "Nikkei_Japan_zscore_60d", "CLX_Clorox_vol_20d", "IYM_BasicMaterials_ret_20d", "XLV_Health_zscore_60d", "CPB_CampbellSoup_zscore_60d", "AMZN_ret_5d", "Core_CPI_zscore_60d", "HangSeng_HK_ret_1d", "INTC_ret_5d", "EOG_EOGResources_ret_5d", "US3Y_Rate_ret_5d", "PAYX_Paychex_ret_20d", "CPB_CampbellSoup_vol_20d", "PCAR_PaccarInc_ret_5d", "EWG_Germany_ret_20d", "AMD_ret_1d", "hmm_p_stress", "EWA_Australia_ret_1d", "CPB_CampbellSoup_ret_5d", "BTI_BritishAmerican_ret_5d"], "is_new": true}, {"model_id": "new_h5_GLOBAL_GradientBoosting_N30_t7", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 5, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "US3M_Rate_vol_20d", "EWA_Australia_ret_1d", "Brent_Oil_FRED_ret_20d", "heston_ev_h3", "3M_ret_5d", "NFCI_ret_5d", "CPB_CampbellSoup_vol_20d", "SBUX_ret_5d", "EWM_Malaysia_ret_1d", "JNJ_ret_1d", "LLY_zscore_60d", "XLY_Disc_vol_20d", "XLV_Health_zscore_60d", "SBUX_zscore_60d", "DAX_Germany_vol_20d", "Retail_Sales_zscore_60d", "XLK_Tech_zscore_60d", "PAYX_Paychex_ret_20d", "EFFR_vol_20d", "Core_CPI_zscore_60d", "Nikkei_Japan_vol_20d", "MSTR_Bitcoin3_ret_20d", "HD_ret_5d", "BLK_BlackRock_zscore_60d", "CCI_CrownCastle_vol_20d", "MS_MorganStanley_ret_1d", "EQIX_Equinix_ret_5d", "ITT_ITTInc_ret_5d"], "is_new": true}, {"model_id": "new_h5_GLOBAL_RandomForest_N5_t0", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 5, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "LOW_Lowes_ret_5d", "GD_GeneralDynamics_zscore_60d", "PCAR_PaccarInc_ret_5d"], "is_new": true}, {"model_id": "new_h5_GLOBAL_RandomForest_N5_t1", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 5, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "SJM_JM_Smucker_ret_5d", "VVIX_ret_20d", "EWA_Australia_zscore_60d"], "is_new": true}, {"model_id": "new_h5_GLOBAL_RandomForest_N5_t2", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 5, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "Nikkei_Japan_vol_20d", "CPB_CampbellSoup_zscore_60d", "MS_MorganStanley_ret_5d"], "is_new": true}, {"model_id": "new_h5_GLOBAL_RandomForest_N5_t3", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 5, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "AMGN_Amgen_ret_1d", "CPB_CampbellSoup_zscore_60d", "HangSeng_HK_vol_20d"], "is_new": true}, {"model_id": "new_h5_GLOBAL_RandomForest_N5_t4", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 5, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "SJM_JM_Smucker_ret_1d", "AORD_AUS_zscore_60d", "EWQ_France_zscore_60d"], "is_new": true}, {"model_id": "new_h5_GLOBAL_RandomForest_N5_t5", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 5, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "QQQ_vol_20d", "MS_MorganStanley_zscore_60d", "BTI_BritishAmerican_ret_5d"], "is_new": true}, {"model_id": "new_h5_GLOBAL_RandomForest_N5_t6", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 5, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "SJM_JM_Smucker_ret_1d", "EOG_EOGResources_vol_20d", "EWA_Australia_ret_1d"], "is_new": true}, {"model_id": "new_h5_GLOBAL_RandomForest_N5_t7", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 5, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWC_Canada_zscore_60d", "IWM_SmallCap_vol_20d", "HangSeng_HK_vol_20d"], "is_new": true}, {"model_id": "new_h5_GLOBAL_RandomForest_N8_t0", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 5, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "INTC_ret_5d", "EOG_EOGResources_vol_20d", "Nikkei_Japan_zscore_60d", "spx_momentum_3d", "AMD_ret_1d", "SBUX_zscore_60d"], "is_new": true}, {"model_id": "new_h5_GLOBAL_RandomForest_N8_t1", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 5, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "SO_SouthernCo_ret_5d", "T10Y2Y_Spread_ret_5d", "EWA_Australia_ret_1d", "US3Y_Rate_ret_5d", "SBUX_vol_20d", "US1Y_Rate_ret_20d"], "is_new": true}, {"model_id": "new_h5_GLOBAL_RandomForest_N8_t2", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 5, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "M_Macys_vol_20d", "ITT_ITTInc_ret_5d", "US3Y_Rate_ret_5d", "heston_var_ev_h5", "CCI_CrownCastle_vol_20d", "HD_ret_20d"], "is_new": true}, {"model_id": "new_h5_GLOBAL_RandomForest_N8_t3", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 5, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWM_Malaysia_ret_1d", "EQIX_Equinix_ret_5d", "EWC_Canada_zscore_60d", "DAX_Germany_vol_20d", "DOW_Price_zscore_60d", "FedFunds_zscore_60d"], "is_new": true}, {"model_id": "new_h5_GLOBAL_RandomForest_N8_t4", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 5, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "SBUX_zscore_60d", "FedFunds_zscore_60d", "DAX_Germany_zscore_60d", "IYM_BasicMaterials_ret_20d", "SLB_Schlumberger_ret_1d", "US3M_Rate_vol_20d"], "is_new": true}, {"model_id": "new_h5_GLOBAL_RandomForest_N8_t5", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 5, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "HangSeng_HK_ret_1d", "IYM_BasicMaterials_ret_20d", "TM_Telephone_vol_20d", "DOW_Price_zscore_60d", "SBUX_ret_5d", "EWL_Switzerland_vol_20d"], "is_new": true}, {"model_id": "new_h5_GLOBAL_RandomForest_N8_t6", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 5, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "Core_CPI_zscore_60d", "ORCL_vol_20d", "SLB_Schlumberger_ret_1d", "VOD_Vodafone_zscore_60d", "EWG_Germany_ret_20d", "XLF_Fin_vol_20d"], "is_new": true}, {"model_id": "new_h5_GLOBAL_RandomForest_N8_t7", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 5, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "Core_PCE_zscore_60d", "TED_Spread_vol_20d", "INTC_ret_5d", "AMGN_Amgen_ret_1d", "HD_zscore_60d", "VOD_Vodafone_zscore_60d"], "is_new": true}, {"model_id": "new_h5_GLOBAL_RandomForest_N10_t0", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 5, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "MS_MorganStanley_zscore_60d", "SCHW_Schwab_ret_5d", "AMD_ret_5d", "TED_Spread_vol_20d", "M_Macys_vol_20d", "DAX_Germany_zscore_60d", "GD_GeneralDynamics_zscore_60d", "AORD_AUS_zscore_60d"], "is_new": true}, {"model_id": "new_h5_GLOBAL_RandomForest_N10_t1", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 5, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "gjr_condvar_h1", "BA_ret_1d", "AORD_AUS_zscore_60d", "US5Y_Rate_ret_5d", "EQR_Equity_ret_1d", "PPL_PPL_ret_1d", "EWL_Switzerland_vol_20d", "EWM_Malaysia_zscore_60d"], "is_new": true}, {"model_id": "new_h5_GLOBAL_RandomForest_N10_t2", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 5, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "INTC_ret_5d", "SCHW_Schwab_ret_5d", "AMZN_ret_5d", "DHR_vol_20d", "M_Macys_vol_20d", "SJM_JM_Smucker_ret_1d", "EFFR_vol_20d", "BTI_BritishAmerican_ret_5d"], "is_new": true}, {"model_id": "new_h5_GLOBAL_RandomForest_N10_t3", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 5, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "Brent_Oil_FRED_ret_5d", "LOW_Lowes_ret_5d", "PAYX_Paychex_zscore_60d", "SBUX_zscore_60d", "EWM_Malaysia_zscore_60d", "DAX_Germany_vol_20d", "Retail_Sales_zscore_60d", "IYM_BasicMaterials_ret_20d"], "is_new": true}, {"model_id": "new_h5_GLOBAL_RandomForest_N10_t4", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 5, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "TM_Telephone_ret_1d", "MSTR_Bitcoin3_ret_20d", "spx_abs_ret_max_5d", "VVIX_ret_20d", "AVB_AvalonBay_zscore_60d", "EWL_Switzerland_zscore_60d", "MS_MorganStanley_ret_5d", "Core_CPI_zscore_60d"], "is_new": true}, {"model_id": "new_h5_GLOBAL_RandomForest_N10_t5", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 5, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "TM_Telephone_ret_1d", "ENB_EnbridgeInc_ret_1d", "EFFR_vol_20d", "US7Y_Rate_ret_20d", "SO_SouthernCo_ret_5d", "spx_abs_ret_max_5d", "EWA_Australia_ret_1d", "EWH_HongKong_ret_5d"], "is_new": true}, {"model_id": "new_h5_GLOBAL_RandomForest_N10_t6", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 5, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "CPB_CampbellSoup_zscore_60d", "QQQ_vol_20d", "T_ret_1d", "EWH_HongKong_ret_5d", "AMD_ret_1d", "BA_ret_1d", "EWA_Australia_zscore_60d", "AORD_AUS_zscore_60d"], "is_new": true}, {"model_id": "new_h5_GLOBAL_RandomForest_N10_t7", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 5, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWY_Korea_zscore_60d", "CCI_CrownCastle_vol_20d", "ITT_ITTInc_ret_5d", "US1Y_Rate_ret_20d", "VVIX_ret_20d", "Michigan_Sentiment_ret_20d", "T_ret_1d", "US6M_Rate_ret_20d"], "is_new": true}, {"model_id": "new_h5_GLOBAL_RandomForest_N12_t0", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 5, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "spx_momentum_3d", "TXN_vol_20d", "PCAR_PaccarInc_ret_5d", "INTC_ret_5d", "heston_var_ev_h5", "TM_Telephone_ret_1d", "EWH_HongKong_ret_5d", "EWA_Australia_zscore_60d", "CPB_CampbellSoup_ret_20d", "spx_abs_ret_max_5d"], "is_new": true}, {"model_id": "new_h5_GLOBAL_RandomForest_N12_t1", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 5, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "IWM_SmallCap_vol_20d", "SBUX_zscore_60d", "EWQ_France_zscore_60d", "heston_var_ev_h3", "EFFR_ret_1d", "3M_vol_20d", "TGT_Target_zscore_60d", "EWH_HongKong_ret_5d", "NFCI_ret_5d", "CPB_CampbellSoup_ret_5d"], "is_new": true}, {"model_id": "new_h5_GLOBAL_RandomForest_N12_t2", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 5, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "SJM_JM_Smucker_ret_5d", "PPL_PPL_ret_1d", "VOD_Vodafone_zscore_60d", "HD_ret_5d", "EWM_Malaysia_zscore_60d", "CLX_Clorox_vol_20d", "BTI_BritishAmerican_ret_20d", "AMD_ret_5d", "VRP_ma5", "SLB_Schlumberger_ret_1d"], "is_new": true}, {"model_id": "new_h5_GLOBAL_RandomForest_N12_t3", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 5, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "NEE_NextEra_ret_20d", "CPB_CampbellSoup_ret_20d", "BTI_BritishAmerican_ret_20d", "EWA_Australia_ret_1d", "EWY_Korea_ret_20d", "Nikkei_Japan_vol_20d", "EOG_EOGResources_vol_20d", "US1Y_Rate_ret_5d", "T_ret_1d", "MO_AltriaMG_ret_1d"], "is_new": true}, {"model_id": "new_h5_GLOBAL_RandomForest_N12_t4", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 5, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "TXN_vol_20d", "XLB_Materials_zscore_60d", "NEE_NextEra_ret_20d", "PPL_PPL_ret_1d", "CPB_CampbellSoup_zscore_60d", "US3M_Rate_zscore_60d", "NOC_Northrop_ret_20d", "LMT_LockheedMartin_ret_1d", "EFFR_vol_20d", "heston_var_ev_h5"], "is_new": true}, {"model_id": "new_h5_GLOBAL_RandomForest_N12_t5", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 5, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "vix_mean_abs_ret_5d", "NEE_NextEra_ret_20d", "NVDA_vol_20d", "XLV_Health_zscore_60d", "TED_Spread_vol_20d", "AXP_Amex_ret_20d", "EWG_Germany_vol_20d", "DHR_ret_1d", "heston_var_ev_h5", "PAYX_Paychex_ret_20d"], "is_new": true}, {"model_id": "new_h5_GLOBAL_RandomForest_N12_t6", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 5, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "MSTR_Bitcoin3_ret_5d", "EQIX_Equinix_ret_5d", "US5Y_Rate_ret_5d", "AMT_AmericanTower_ret_1d", "BTI_BritishAmerican_ret_5d", "EWG_Germany_vol_20d", "PCAR_PaccarInc_ret_5d", "HD_ret_20d", "EFFR_vol_20d", "TXN_vol_20d"], "is_new": true}, {"model_id": "new_h5_GLOBAL_RandomForest_N12_t7", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 5, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "vix_acceleration_1d", "MSTR_Bitcoin3_ret_1d", "SBUX_vol_20d", "EWJ_Japan_vol_20d", "Industrial_Production_zscore_60d", "AXP_Amex_ret_20d", "TGT_Target_zscore_60d", "EOG_EOGResources_vol_20d", "DE_Deere_ret_5d", "AMD_ret_1d"], "is_new": true}, {"model_id": "new_h5_GLOBAL_RandomForest_N15_t0", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 5, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "AMT_AmericanTower_ret_1d", "CPB_CampbellSoup_ret_20d", "VRP_ma5", "DE_Deere_ret_5d", "FedFunds_zscore_60d", "US5Y_Rate_ret_5d", "AORD_AUS_zscore_60d", "PAYX_Paychex_vol_20d", "HD_ret_1d", "SLB_Schlumberger_ret_5d", "TED_Spread_zscore_60d", "EQIX_Equinix_ret_5d", "CPB_CampbellSoup_ret_5d"], "is_new": true}, {"model_id": "new_h5_GLOBAL_RandomForest_N15_t1", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 5, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "NVDA_vol_20d", "PLD_Prologis_ret_5d", "DIS_vol_20d", "AORD_AUS_zscore_60d", "QQQ_vol_20d", "BTI_BritishAmerican_ret_20d", "heston_var_ev_h7", "VOD_Vodafone_zscore_60d", "AMGN_Amgen_ret_1d", "CPB_CampbellSoup_zscore_60d", "GE_ret_1d", "SBUX_ret_5d", "EWM_Malaysia_ret_1d"], "is_new": true}, {"model_id": "new_h5_GLOBAL_RandomForest_N15_t2", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 5, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EOG_EOGResources_vol_20d", "AXP_Amex_vol_20d", "SPY_zscore_60d", "US3Y_Rate_ret_5d", "MO_AltriaMG_ret_1d", "TM_Telephone_ret_1d", "AMD_ret_1d", "XLV_Health_zscore_60d", "AMZN_ret_5d", "spx_momentum_3d", "CPB_CampbellSoup_vol_20d", "LOW_Lowes_ret_5d", "Brent_Oil_FRED_ret_5d"], "is_new": true}, {"model_id": "new_h5_GLOBAL_RandomForest_N15_t3", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 5, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "DAX_Germany_zscore_60d", "AMGN_Amgen_ret_1d", "NEE_NextEra_ret_20d", "TM_Telephone_ret_1d", "NVDA_vol_20d", "SBUX_ret_5d", "EWS_Singapore_ret_5d", "GILD_Gilead_ret_20d", "EWQ_France_zscore_60d", "XOM_ret_20d", "TGT_Target_zscore_60d", "VVIX_ret_20d", "GD_GeneralDynamics_zscore_60d"], "is_new": true}, {"model_id": "new_h5_GLOBAL_RandomForest_N15_t4", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 5, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "LOW_Lowes_ret_20d", "VOD_Vodafone_zscore_60d", "T10Y2Y_Spread_ret_5d", "EQR_Equity_ret_1d", "heston_var_ev_h7", "BLK_BlackRock_zscore_60d", "DE_Deere_vol_20d", "US3Y_Rate_ret_5d", "heston_ev_h3", "AVB_AvalonBay_zscore_60d", "BTI_BritishAmerican_ret_5d", "MS_MorganStanley_ret_5d", "M_Macys_vol_20d"], "is_new": true}, {"model_id": "new_h5_GLOBAL_RandomForest_N15_t5", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 5, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "PG_ret_20d", "FedFunds_zscore_60d", "SBUX_zscore_60d", "ORCL_zscore_60d", "EFFR_vol_20d", "EWS_Singapore_ret_5d", "vix_mean_abs_ret_5d", "HD_ret_20d", "EWH_HongKong_ret_5d", "Nikkei_Japan_zscore_60d", "MS_MorganStanley_ret_5d", "XOM_ret_20d", "NVDA_vol_20d"], "is_new": true}, {"model_id": "new_h5_GLOBAL_RandomForest_N15_t6", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 5, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "IYR_US_REIT2_zscore_60d", "hmm_p_stress", "3M_vol_20d", "EWL_Switzerland_vol_20d", "EWQ_France_zscore_60d", "LOW_Lowes_ret_20d", "AMGN_Amgen_ret_1d", "GD_GeneralDynamics_zscore_60d", "MSTR_Bitcoin3_ret_5d", "SLB_Schlumberger_ret_1d", "AXP_Amex_vol_20d", "CPB_CampbellSoup_vol_20d", "ITT_ITTInc_ret_5d"], "is_new": true}, {"model_id": "new_h5_GLOBAL_RandomForest_N15_t7", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 5, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "LMT_LockheedMartin_ret_1d", "ASX_Australia_vol_20d", "XOM_ret_1d", "MS_MorganStanley_ret_5d", "ASX_Australia_ret_5d", "EWY_Korea_ret_20d", "PCAR_PaccarInc_ret_5d", "LOW_Lowes_ret_5d", "SJM_JM_Smucker_ret_5d", "JNJ_ret_1d", "NEE_NextEra_ret_20d", "XLF_Fin_vol_20d", "vix_acceleration_1d"], "is_new": true}, {"model_id": "new_h5_GLOBAL_RandomForest_N20_t0", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 5, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EOG_EOGResources_vol_20d", "VOD_Vodafone_zscore_60d", "XOM_ret_1d", "DAX_Germany_vol_20d", "XLF_Fin_vol_20d", "EXC_Exelon_zscore_60d", "heston_ev_h3", "Industrial_Production_zscore_60d", "SBUX_vol_20d", "MSTR_Bitcoin3_ret_1d", "EWQ_France_ret_20d", "hmm_p_stress", "EXC_Exelon_ret_1d", "EWL_Switzerland_vol_20d", "EWY_Korea_ret_20d", "WTI_Oil_FRED_zscore_60d", "US6M_Rate_ret_20d", "SJM_JM_Smucker_ret_1d"], "is_new": true}, {"model_id": "new_h5_GLOBAL_RandomForest_N20_t1", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 5, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWL_Switzerland_zscore_60d", "ASX_Australia_vol_20d", "ORCL_vol_20d", "Core_PCE_zscore_60d", "XLF_Fin_vol_20d", "EWM_Malaysia_ret_1d", "PAYX_Paychex_ret_20d", "vix_acceleration_1d", "PLD_Prologis_ret_5d", "heston_var_ev_h7", "BLK_BlackRock_zscore_60d", "MS_MorganStanley_zscore_60d", "XLV_Health_zscore_60d", "SLB_Schlumberger_ret_1d", "QQQ_vol_20d", "TGT_Target_zscore_60d", "AXP_Amex_vol_20d", "heston_var_ev_h5"], "is_new": true}, {"model_id": "new_h5_GLOBAL_RandomForest_N20_t2", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 5, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "MSTR_Bitcoin3_ret_5d", "spx_abs_ret_max_5d", "US7Y_Rate_ret_20d", "US3M_Rate_zscore_60d", "EWG_Germany_ret_20d", "GD_GeneralDynamics_zscore_60d", "hmm_p_stress", "CLX_Clorox_vol_20d", "BDX_Becton_Dickinson_ret_20d", "IYM_BasicMaterials_ret_20d", "JNJ_ret_1d", "MSTR_Bitcoin3_ret_20d", "SBUX_vol_20d", "ENB_EnbridgeInc_ret_1d", "FedFunds_zscore_60d", "DAX_Germany_zscore_60d", "PAYX_Paychex_zscore_60d", "GILD_Gilead_ret_20d"], "is_new": true}, {"model_id": "new_h5_GLOBAL_RandomForest_N20_t3", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 5, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EQIX_Equinix_ret_5d", "heston_ev_h3", "PG_ret_20d", "Core_PCE_zscore_60d", "PFE_ret_1d", "vix_mean_abs_ret_5d", "BTI_BritishAmerican_ret_20d", "EWA_Australia_zscore_60d", "ORCL_zscore_60d", "JNJ_ret_1d", "M_Macys_vol_20d", "HangSeng_HK_vol_20d", "US1Y_Rate_ret_5d", "NVDA_vol_20d", "CTAS_Cintas_vol_20d", "FedFunds_zscore_60d", "EWL_Switzerland_zscore_60d", "NFCI_ret_5d"], "is_new": true}, {"model_id": "new_h5_GLOBAL_RandomForest_N20_t4", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 5, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "PG_ret_20d", "INTC_ret_1d", "WTI_Oil_FRED_zscore_60d", "AMD_ret_1d", "SJM_JM_Smucker_ret_5d", "SO_SouthernCo_ret_5d", "XOM_ret_20d", "Retail_Sales_zscore_60d", "AMD_ret_5d", "BLK_BlackRock_zscore_60d", "EFFR_ret_1d", "MO_AltriaMG_ret_1d", "CPB_CampbellSoup_zscore_60d", "TED_Spread_zscore_60d", "US1Y_Rate_ret_5d", "EOG_EOGResources_ret_5d", "CTAS_Cintas_vol_20d", "MS_MorganStanley_ret_5d"], "is_new": true}, {"model_id": "new_h5_GLOBAL_RandomForest_N20_t5", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 5, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWQ_France_ret_20d", "FedFunds_zscore_60d", "HD_zscore_60d", "BTI_BritishAmerican_ret_5d", "Core_CPI_zscore_60d", "3M_ret_5d", "EOG_EOGResources_ret_5d", "EOG_EOGResources_vol_20d", "ASX_Australia_ret_5d", "Brent_Oil_FRED_ret_20d", "EWM_Malaysia_ret_1d", "US3M_Rate_vol_20d", "DE_Deere_vol_20d", "NFCI_ret_5d", "AMZN_ret_5d", "gjr_condvar_h1", "US3Y_Rate_ret_5d", "DHR_ret_1d"], "is_new": true}, {"model_id": "new_h5_GLOBAL_RandomForest_N20_t6", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 5, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "VVIX_ret_20d", "XOM_ret_1d", "SBUX_vol_20d", "EOG_EOGResources_vol_20d", "NVDA_vol_20d", "EWS_Singapore_ret_5d", "EWQ_France_zscore_60d", "DOW_Price_zscore_60d", "US3M_Rate_vol_20d", "EWC_Canada_zscore_60d", "EWL_Switzerland_zscore_60d", "vix_acceleration_1d", "BA_ret_1d", "AMD_ret_5d", "EFFR_ret_1d", "EWG_Germany_vol_20d", "EOG_EOGResources_ret_5d", "TM_Telephone_vol_20d"], "is_new": true}, {"model_id": "new_h5_GLOBAL_RandomForest_N20_t7", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 5, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "DHR_vol_20d", "GD_GeneralDynamics_zscore_60d", "AMD_ret_1d", "AVB_AvalonBay_zscore_60d", "MS_MorganStanley_ret_1d", "DHR_ret_1d", "DE_Deere_ret_5d", "GE_ret_1d", "CPB_CampbellSoup_ret_20d", "SBUX_ret_5d", "HangSeng_HK_ret_5d", "TED_Spread_zscore_60d", "Michigan_Sentiment_ret_20d", "AXP_Amex_ret_20d", "AMT_AmericanTower_ret_1d", "M_Macys_vol_20d", "VRP_ma5", "SO_SouthernCo_ret_5d"], "is_new": true}, {"model_id": "new_h5_GLOBAL_RandomForest_N25_t0", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 5, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "ORCL_vol_20d", "XLF_Fin_vol_20d", "AMZN_ret_5d", "heston_ev_h3", "EOG_EOGResources_ret_5d", "VRP_ma5", "AMD_ret_5d", "EOG_EOGResources_vol_20d", "PG_ret_20d", "M_Macys_vol_20d", "US7Y_Rate_ret_20d", "vix_acceleration_1d", "US6M_Rate_ret_20d", "AMT_AmericanTower_ret_1d", "GD_GeneralDynamics_zscore_60d", "INTC_ret_1d", "AXP_Amex_ret_20d", "US1Y_Rate_ret_5d", "ES_Evergy_ret_1d", "AMGN_Amgen_ret_1d", "spx_abs_ret_max_5d", "Nikkei_Japan_zscore_60d", "DE_Deere_ret_5d"], "is_new": true}, {"model_id": "new_h5_GLOBAL_RandomForest_N25_t1", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 5, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "IYM_BasicMaterials_ret_20d", "EWJ_Japan_vol_20d", "US5Y_Rate_ret_5d", "TED_Spread_vol_20d", "EWA_Australia_ret_1d", "SBUX_zscore_60d", "HD_zscore_60d", "INTC_ret_1d", "EWM_Malaysia_ret_1d", "PAYX_Paychex_ret_20d", "GD_GeneralDynamics_zscore_60d", "BA_ret_1d", "Brent_Oil_FRED_ret_5d", "ES_Evergy_ret_1d", "MSTR_Bitcoin3_ret_5d", "TM_Telephone_vol_20d", "gjr_condvar_h1", "ASX_Australia_vol_20d", "TM_Telephone_ret_1d", "LLY_zscore_60d", "NOC_Northrop_ret_20d", "heston_var_ev_h7", "EWC_Canada_zscore_60d"], "is_new": true}, {"model_id": "new_h5_GLOBAL_RandomForest_N25_t2", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 5, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "XOM_ret_1d", "EWM_Malaysia_vol_20d", "Core_PCE_zscore_60d", "DIS_vol_20d", "FedFunds_zscore_60d", "SPY_zscore_60d", "EWQ_France_zscore_60d", "TGT_Target_zscore_60d", "EQIX_Equinix_ret_5d", "PAYX_Paychex_zscore_60d", "XLF_Fin_vol_20d", "AXP_Amex_vol_20d", "MS_MorganStanley_ret_1d", "INTC_ret_1d", "HangSeng_HK_ret_5d", "Industrial_Production_zscore_60d", "heston_ev_h3", "EWQ_France_ret_20d", "spx_abs_ret_max_5d", "US1Y_Rate_ret_5d", "IYM_BasicMaterials_ret_20d", "GE_ret_1d", "spx_vol_5d"], "is_new": true}, {"model_id": "new_h5_GLOBAL_RandomForest_N25_t3", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 5, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "MSTR_Bitcoin3_ret_20d", "Brent_Oil_FRED_ret_20d", "NOC_Northrop_ret_20d", "EQR_Equity_ret_1d", "MO_AltriaMG_ret_1d", "spx_momentum_3d", "EWM_Malaysia_ret_1d", "ASX_Australia_ret_5d", "US1Y_Rate_ret_20d", "BTI_BritishAmerican_ret_5d", "VVIX_ret_20d", "PG_ret_20d", "EWL_Switzerland_vol_20d", "EWA_Australia_zscore_60d", "FedFunds_zscore_60d", "HangSeng_HK_ret_1d", "IYR_US_REIT2_zscore_60d", "DIS_vol_20d", "AMD_ret_5d", "EWY_Korea_zscore_60d", "LOW_Lowes_ret_20d", "XLB_Materials_zscore_60d", "HD_ret_1d"], "is_new": true}, {"model_id": "new_h5_GLOBAL_RandomForest_N25_t4", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 5, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "NFCI_ret_5d", "SBUX_zscore_60d", "EWG_Germany_ret_20d", "SPY_zscore_60d", "heston_var_ev_h7", "SLB_Schlumberger_ret_5d", "XLK_Tech_zscore_60d", "AMD_ret_5d", "PAYX_Paychex_zscore_60d", "MSTR_Bitcoin3_ret_1d", "Core_CPI_zscore_60d", "EWY_Korea_ret_20d", "Core_PCE_zscore_60d", "T_ret_1d", "QQQ_vol_20d", "MSTR_Bitcoin3_ret_5d", "EWM_Malaysia_vol_20d", "Industrial_Production_zscore_60d", "HD_zscore_60d", "AVB_AvalonBay_zscore_60d", "AMZN_ret_5d", "EOG_EOGResources_ret_5d", "DHR_ret_1d"], "is_new": true}, {"model_id": "new_h5_GLOBAL_RandomForest_N25_t5", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 5, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "US30Y_Rate_ret_20d", "US1Y_Rate_ret_5d", "NVDA_vol_20d", "US7Y_Rate_ret_20d", "LOW_Lowes_ret_5d", "CPB_CampbellSoup_vol_20d", "EFFR_vol_20d", "Retail_Sales_zscore_60d", "3M_ret_5d", "INTC_ret_1d", "SBUX_zscore_60d", "AORD_AUS_zscore_60d", "heston_var_ev_h7", "SBUX_ret_5d", "EWA_Australia_ret_1d", "AMT_AmericanTower_ret_1d", "EWM_Malaysia_zscore_60d", "SPY_zscore_60d", "EQR_Equity_ret_1d", "BLK_BlackRock_zscore_60d", "BA_ret_1d", "AMD_ret_1d", "ASX_Australia_vol_20d"], "is_new": true}, {"model_id": "new_h5_GLOBAL_RandomForest_N25_t6", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 5, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "BLK_BlackRock_zscore_60d", "US30Y_Rate_ret_20d", "AXP_Amex_vol_20d", "spx_vol_5d", "US3M_Rate_zscore_60d", "EOG_EOGResources_vol_20d", "vix_acceleration_1d", "EOG_EOGResources_ret_5d", "CTAS_Cintas_vol_20d", "3M_ret_5d", "EWC_Canada_zscore_60d", "TXN_vol_20d", "TED_Spread_zscore_60d", "SBUX_ret_5d", "SLB_Schlumberger_ret_5d", "CPB_CampbellSoup_vol_20d", "IYR_US_REIT2_zscore_60d", "Core_CPI_zscore_60d", "HD_zscore_60d", "XLK_Tech_zscore_60d", "PLD_Prologis_ret_5d", "SJM_JM_Smucker_ret_5d", "US6M_Rate_ret_20d"], "is_new": true}, {"model_id": "new_h5_GLOBAL_RandomForest_N25_t7", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 5, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "BDX_Becton_Dickinson_ret_20d", "CCI_CrownCastle_vol_20d", "MS_MorganStanley_ret_5d", "EQIX_Equinix_ret_5d", "HD_ret_20d", "DHR_ret_1d", "VVIX_ret_20d", "LMT_LockheedMartin_vol_20d", "LUV_SouthwestAir_ret_5d", "MO_AltriaMG_ret_1d", "EWL_Switzerland_zscore_60d", "SLB_Schlumberger_ret_5d", "SBUX_ret_5d", "AXP_Amex_vol_20d", "US3Y_Rate_ret_5d", "Retail_Sales_zscore_60d", "US6M_Rate_ret_20d", "CLX_Clorox_vol_20d", "INTC_ret_5d", "HangSeng_HK_ret_5d", "EFFR_vol_20d", "EOG_EOGResources_vol_20d", "BA_ret_1d"], "is_new": true}, {"model_id": "new_h5_GLOBAL_RandomForest_N30_t0", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 5, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "PPL_PPL_ret_1d", "BTI_BritishAmerican_ret_20d", "MRK_Merck_zscore_60d", "TED_Spread_zscore_60d", "SBUX_ret_5d", "IYM_BasicMaterials_ret_20d", "IBEX_Spain_ret_20d", "EXC_Exelon_ret_1d", "XLK_Tech_zscore_60d", "Michigan_Sentiment_ret_20d", "EWJ_Japan_vol_20d", "EQR_Equity_ret_1d", "FedFunds_zscore_60d", "DOW_Price_zscore_60d", "HD_zscore_60d", "MS_MorganStanley_ret_5d", "heston_ev_h3", "BDX_Becton_Dickinson_ret_20d", "heston_var_ev_h7", "SBUX_vol_20d", "XOM_ret_20d", "heston_var_ev_h5", "SLB_Schlumberger_ret_5d", "GD_GeneralDynamics_zscore_60d", "EFFR_ret_1d", "ES_Evergy_ret_1d", "AMZN_ret_5d", "AMT_AmericanTower_ret_1d"], "is_new": true}, {"model_id": "new_h5_GLOBAL_RandomForest_N30_t1", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 5, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "CPB_CampbellSoup_ret_5d", "AVB_AvalonBay_zscore_60d", "BTI_BritishAmerican_ret_5d", "ORCL_vol_20d", "heston_var_ev_h5", "BLK_BlackRock_zscore_60d", "XLB_Materials_zscore_60d", "PAYX_Paychex_ret_20d", "AMZN_ret_5d", "Nikkei_Japan_zscore_60d", "PFE_ret_1d", "AXP_Amex_ret_20d", "EXC_Exelon_ret_1d", "EWS_Singapore_ret_5d", "AMD_ret_1d", "EMR_Emerson_ret_20d", "GE_ret_1d", "CPB_CampbellSoup_zscore_60d", "EWL_Switzerland_vol_20d", "LMT_LockheedMartin_ret_1d", "ASX_Australia_ret_5d", "SJM_JM_Smucker_ret_5d", "AMD_ret_5d", "MO_AltriaMG_ret_1d", "GD_GeneralDynamics_zscore_60d", "EOG_EOGResources_vol_20d", "EWG_Germany_vol_20d", "EWL_Switzerland_zscore_60d"], "is_new": true}, {"model_id": "new_h5_GLOBAL_RandomForest_N30_t2", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 5, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "LOW_Lowes_ret_20d", "Core_PCE_zscore_60d", "T_ret_1d", "DAX_Germany_vol_20d", "EWQ_France_ret_20d", "EWY_Korea_zscore_60d", "EQIX_Equinix_ret_5d", "TM_Telephone_vol_20d", "heston_var_ev_h3", "SO_SouthernCo_ret_5d", "CTAS_Cintas_vol_20d", "LOW_Lowes_ret_5d", "CI_Cigna_vol_20d", "HD_ret_1d", "PAYX_Paychex_ret_20d", "EWQ_France_zscore_60d", "US3M_Rate_zscore_60d", "US5Y_Rate_ret_5d", "AMD_ret_1d", "IWM_SmallCap_vol_20d", "EWM_Malaysia_ret_1d", "PG_ret_20d", "US6M_Rate_ret_20d", "US7Y_Rate_ret_20d", "XLF_Fin_vol_20d", "CPB_CampbellSoup_zscore_60d", "LLY_zscore_60d", "IYR_US_REIT2_zscore_60d"], "is_new": true}, {"model_id": "new_h5_GLOBAL_RandomForest_N30_t3", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 5, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "T_ret_1d", "HangSeng_HK_ret_5d", "heston_var_ev_h7", "CPB_CampbellSoup_ret_5d", "T10Y2Y_Spread_ret_5d", "CPB_CampbellSoup_vol_20d", "NWL_Newell_ret_20d", "DAX_Germany_zscore_60d", "US1Y_Rate_ret_5d", "ES_Evergy_ret_1d", "MS_MorganStanley_ret_1d", "Core_CPI_zscore_60d", "EWC_Canada_zscore_60d", "MS_MorganStanley_ret_5d", "DHR_vol_20d", "EXC_Exelon_zscore_60d", "EWM_Malaysia_zscore_60d", "HD_zscore_60d", "Core_PCE_zscore_60d", "US5Y_Rate_ret_5d", "CI_Cigna_vol_20d", "PG_ret_20d", "XOM_ret_1d", "US3Y_Rate_ret_5d", "INTC_ret_5d", "US3M_Rate_vol_20d", "EWM_Malaysia_vol_20d", "DOW_Price_zscore_60d"], "is_new": true}, {"model_id": "new_h5_GLOBAL_RandomForest_N30_t4", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 5, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "AORD_AUS_zscore_60d", "IYR_US_REIT2_zscore_60d", "INTC_ret_5d", "GD_GeneralDynamics_zscore_60d", "gjr_condvar_h1", "MSTR_Bitcoin3_ret_5d", "EWQ_France_ret_20d", "DHR_ret_1d", "CLX_Clorox_vol_20d", "US1Y_Rate_ret_20d", "EQR_Equity_ret_1d", "PAYX_Paychex_ret_20d", "AMT_AmericanTower_ret_1d", "AMD_ret_1d", "MS_MorganStanley_ret_5d", "heston_var_ev_h5", "Nikkei_Japan_vol_20d", "heston_ev_h3", "NVDA_vol_20d", "CPB_CampbellSoup_ret_20d", "AVB_AvalonBay_zscore_60d", "US1Y_Rate_ret_5d", "HangSeng_HK_ret_1d", "US7Y_Rate_ret_20d", "PPL_PPL_ret_1d", "WTI_Oil_FRED_zscore_60d", "ASX_Australia_vol_20d", "SBUX_zscore_60d"], "is_new": true}, {"model_id": "new_h5_GLOBAL_RandomForest_N30_t5", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 5, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "SJM_JM_Smucker_ret_1d", "HD_zscore_60d", "EWL_Switzerland_vol_20d", "EWM_Malaysia_zscore_60d", "EWC_Canada_zscore_60d", "XLY_Disc_vol_20d", "heston_var_ev_h5", "US1Y_Rate_ret_20d", "XLB_Materials_zscore_60d", "T10Y2Y_Spread_ret_5d", "JNJ_ret_1d", "EFFR_ret_1d", "INTC_ret_5d", "US1Y_Rate_ret_5d", "SO_SouthernCo_ret_5d", "BDX_Becton_Dickinson_ret_20d", "HangSeng_HK_vol_20d", "US3Y_Rate_ret_5d", "MS_MorganStanley_zscore_60d", "EMR_Emerson_ret_20d", "LOW_Lowes_ret_20d", "ENB_EnbridgeInc_ret_1d", "LMT_LockheedMartin_vol_20d", "ITT_ITTInc_ret_5d", "hmm_p_stress", "AORD_AUS_zscore_60d", "EWY_Korea_ret_20d", "CTAS_Cintas_vol_20d"], "is_new": true}, {"model_id": "new_h5_GLOBAL_RandomForest_N30_t6", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 5, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "BTI_BritishAmerican_ret_20d", "EWA_Australia_zscore_60d", "Brent_Oil_FRED_ret_20d", "EWJ_Japan_vol_20d", "LOW_Lowes_ret_20d", "DE_Deere_ret_5d", "BTI_BritishAmerican_ret_5d", "HangSeng_HK_ret_5d", "SLB_Schlumberger_ret_1d", "IYR_US_REIT2_zscore_60d", "SBUX_ret_5d", "Industrial_Production_zscore_60d", "Brent_Oil_FRED_ret_5d", "PAYX_Paychex_zscore_60d", "GD_GeneralDynamics_zscore_60d", "EWQ_France_ret_20d", "vix_mean_abs_ret_5d", "ES_Evergy_ret_1d", "NFCI_ret_5d", "DAX_Germany_zscore_60d", "CCI_CrownCastle_vol_20d", "VOD_Vodafone_zscore_60d", "PAYX_Paychex_vol_20d", "EWY_Korea_ret_20d", "CPB_CampbellSoup_vol_20d", "HangSeng_HK_ret_1d", "LUV_SouthwestAir_ret_5d", "XLY_Disc_vol_20d"], "is_new": true}, {"model_id": "new_h5_GLOBAL_RandomForest_N30_t7", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 5, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "ES_Evergy_ret_1d", "ITT_ITTInc_ret_5d", "SO_SouthernCo_ret_5d", "spx_momentum_3d", "NWL_Newell_ret_20d", "MS_MorganStanley_ret_5d", "3M_ret_5d", "Core_PCE_zscore_60d", "US3M_Rate_zscore_60d", "FedFunds_zscore_60d", "M_Macys_vol_20d", "CPB_CampbellSoup_vol_20d", "heston_var_ev_h5", "spx_abs_ret_max_5d", "heston_var_ev_h7", "TM_Telephone_ret_1d", "Core_CPI_zscore_60d", "AMGN_Amgen_ret_1d", "SBUX_vol_20d", "HangSeng_HK_ret_1d", "Industrial_Production_zscore_60d", "EFFR_vol_20d", "LMT_LockheedMartin_vol_20d", "HangSeng_HK_ret_5d", "TXN_vol_20d", "Michigan_Sentiment_ret_20d", "XOM_ret_1d", "IYM_BasicMaterials_ret_20d"], "is_new": true}, {"model_id": "new_h5_GLOBAL_LogisticRegression_N5_t0", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 5, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "CMCSA_ret_1d", "AXP_Amex_vol_20d", "heston_ev_h3"], "is_new": true}, {"model_id": "new_h5_GLOBAL_LogisticRegression_N5_t1", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 5, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "M_Macys_vol_20d", "Core_CPI_zscore_60d", "LLY_zscore_60d"], "is_new": true}, {"model_id": "new_h5_GLOBAL_LogisticRegression_N5_t2", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 5, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "INTC_ret_5d", "PG_ret_20d", "TM_Telephone_ret_1d"], "is_new": true}, {"model_id": "new_h5_GLOBAL_LogisticRegression_N5_t3", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 5, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "CPB_CampbellSoup_vol_20d", "EWG_Germany_vol_20d", "CTAS_Cintas_vol_20d"], "is_new": true}, {"model_id": "new_h5_GLOBAL_LogisticRegression_N5_t4", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 5, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "Brent_Oil_FRED_ret_5d", "AXP_Amex_ret_20d", "BLK_BlackRock_zscore_60d"], "is_new": true}, {"model_id": "new_h5_GLOBAL_LogisticRegression_N5_t5", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 5, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "AMD_ret_1d", "ORCL_vol_20d", "EWH_HongKong_ret_5d"], "is_new": true}, {"model_id": "new_h5_GLOBAL_LogisticRegression_N5_t6", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 5, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "US7Y_Rate_ret_20d", "GE_ret_1d", "vix_mean_abs_ret_5d"], "is_new": true}, {"model_id": "new_h5_GLOBAL_LogisticRegression_N5_t7", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 5, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "US3Y_Rate_ret_5d", "heston_ev_h3", "Industrial_Production_zscore_60d"], "is_new": true}, {"model_id": "new_h5_GLOBAL_LogisticRegression_N8_t0", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 5, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "GD_GeneralDynamics_zscore_60d", "EQR_Equity_ret_1d", "QQQ_vol_20d", "INTC_ret_1d", "vix_acceleration_1d", "BLK_BlackRock_zscore_60d"], "is_new": true}, {"model_id": "new_h5_GLOBAL_LogisticRegression_N8_t1", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 5, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "heston_var_ev_h3", "CPB_CampbellSoup_vol_20d", "Industrial_Production_zscore_60d", "EWC_Canada_zscore_60d", "XOM_ret_20d", "MS_MorganStanley_ret_5d"], "is_new": true}, {"model_id": "new_h5_GLOBAL_LogisticRegression_N8_t2", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 5, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "MSTR_Bitcoin3_ret_20d", "ENB_EnbridgeInc_ret_1d", "EWL_Switzerland_zscore_60d", "AMD_ret_5d", "AMZN_ret_5d", "EWY_Korea_ret_20d"], "is_new": true}, {"model_id": "new_h5_GLOBAL_LogisticRegression_N8_t3", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 5, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "hmm_p_stress", "EOG_EOGResources_ret_5d", "AXP_Amex_ret_20d", "TED_Spread_vol_20d", "EFFR_vol_20d", "EWG_Germany_ret_20d"], "is_new": true}, {"model_id": "new_h5_GLOBAL_LogisticRegression_N8_t4", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 5, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "BTI_BritishAmerican_ret_5d", "CLX_Clorox_vol_20d", "HD_zscore_60d", "VRP_ma5", "Michigan_Sentiment_ret_20d", "EWC_Canada_zscore_60d"], "is_new": true}, {"model_id": "new_h5_GLOBAL_LogisticRegression_N8_t5", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 5, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "TM_Telephone_vol_20d", "EXC_Exelon_ret_1d", "FedFunds_zscore_60d", "GILD_Gilead_ret_20d", "BTI_BritishAmerican_ret_20d", "spx_momentum_3d"], "is_new": true}, {"model_id": "new_h5_GLOBAL_LogisticRegression_N8_t6", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 5, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "BA_ret_1d", "MSTR_Bitcoin3_ret_20d", "DIS_vol_20d", "DHR_vol_20d", "XOM_ret_20d", "CTAS_Cintas_vol_20d"], "is_new": true}, {"model_id": "new_h5_GLOBAL_LogisticRegression_N8_t7", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 5, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "3M_ret_5d", "EWC_Canada_zscore_60d", "DOW_Price_zscore_60d", "AMD_ret_5d", "3M_vol_20d", "EWJ_Japan_vol_20d"], "is_new": true}, {"model_id": "new_h5_GLOBAL_LogisticRegression_N10_t0", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 5, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "IYR_US_REIT2_zscore_60d", "IBEX_Spain_ret_20d", "DE_Deere_vol_20d", "LMT_LockheedMartin_vol_20d", "XLF_Fin_vol_20d", "EFFR_ret_1d", "SLB_Schlumberger_ret_1d", "EWY_Korea_ret_20d"], "is_new": true}, {"model_id": "new_h5_GLOBAL_LogisticRegression_N10_t1", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 5, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "Core_CPI_zscore_60d", "ASX_Australia_ret_5d", "IYR_US_REIT2_zscore_60d", "VRP_ma5", "SCHW_Schwab_ret_5d", "AMD_ret_5d", "CI_Cigna_vol_20d", "ES_Evergy_ret_1d"], "is_new": true}, {"model_id": "new_h5_GLOBAL_LogisticRegression_N10_t2", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 5, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWC_Canada_zscore_60d", "MS_MorganStanley_zscore_60d", "3M_vol_20d", "DHR_vol_20d", "INTC_ret_5d", "NOC_Northrop_ret_20d", "SBUX_zscore_60d", "EWQ_France_ret_20d"], "is_new": true}, {"model_id": "new_h5_GLOBAL_LogisticRegression_N10_t3", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 5, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "CPB_CampbellSoup_vol_20d", "ENB_EnbridgeInc_ret_1d", "EWY_Korea_ret_20d", "ES_Evergy_ret_1d", "BA_ret_1d", "WTI_Oil_FRED_zscore_60d", "ITT_ITTInc_ret_5d", "IYM_BasicMaterials_ret_20d"], "is_new": true}, {"model_id": "new_h5_GLOBAL_LogisticRegression_N10_t4", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 5, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWM_Malaysia_vol_20d", "MSTR_Bitcoin3_ret_1d", "SLB_Schlumberger_ret_5d", "heston_var_ev_h3", "TM_Telephone_vol_20d", "NOC_Northrop_ret_20d", "US3Y_Rate_ret_5d", "CPB_CampbellSoup_zscore_60d"], "is_new": true}, {"model_id": "new_h5_GLOBAL_LogisticRegression_N10_t5", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 5, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "MS_MorganStanley_ret_5d", "NEE_NextEra_ret_20d", "MRK_Merck_zscore_60d", "MS_MorganStanley_zscore_60d", "US5Y_Rate_ret_5d", "EWC_Canada_zscore_60d", "AMT_AmericanTower_ret_1d", "AMZN_ret_5d"], "is_new": true}, {"model_id": "new_h5_GLOBAL_LogisticRegression_N10_t6", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 5, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "INTC_ret_5d", "heston_var_ev_h5", "IYM_BasicMaterials_ret_20d", "NOC_Northrop_ret_20d", "PAYX_Paychex_zscore_60d", "MSTR_Bitcoin3_ret_5d", "CCI_CrownCastle_vol_20d", "EWM_Malaysia_zscore_60d"], "is_new": true}, {"model_id": "new_h5_GLOBAL_LogisticRegression_N10_t7", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 5, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWA_Australia_zscore_60d", "GILD_Gilead_ret_20d", "GE_ret_1d", "EWQ_France_ret_20d", "AMT_AmericanTower_ret_1d", "EWY_Korea_zscore_60d", "QQQ_vol_20d", "CMCSA_ret_1d"], "is_new": true}, {"model_id": "new_h5_GLOBAL_LogisticRegression_N12_t0", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 5, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "AMD_ret_1d", "LLY_zscore_60d", "CCI_CrownCastle_vol_20d", "EWA_Australia_zscore_60d", "XLB_Materials_zscore_60d", "LOW_Lowes_ret_20d", "HD_ret_20d", "CPB_CampbellSoup_ret_5d", "HD_zscore_60d", "US3M_Rate_zscore_60d"], "is_new": true}, {"model_id": "new_h5_GLOBAL_LogisticRegression_N12_t1", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 5, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "CCI_CrownCastle_vol_20d", "DOW_Price_zscore_60d", "TED_Spread_zscore_60d", "EXC_Exelon_ret_1d", "DIS_vol_20d", "INTC_ret_5d", "NFCI_ret_5d", "heston_var_ev_h5", "CMCSA_ret_1d", "VVIX_ret_20d"], "is_new": true}, {"model_id": "new_h5_GLOBAL_LogisticRegression_N12_t2", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 5, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "heston_ev_h3", "Michigan_Sentiment_ret_20d", "TGT_Target_zscore_60d", "NVDA_vol_20d", "DHR_vol_20d", "MO_AltriaMG_ret_1d", "CLX_Clorox_vol_20d", "EWA_Australia_zscore_60d", "PLD_Prologis_ret_5d", "Retail_Sales_zscore_60d"], "is_new": true}, {"model_id": "new_h5_GLOBAL_LogisticRegression_N12_t3", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 5, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "LMT_LockheedMartin_vol_20d", "LOW_Lowes_ret_5d", "LOW_Lowes_ret_20d", "WTI_Oil_FRED_zscore_60d", "AMD_ret_1d", "SO_SouthernCo_ret_5d", "HD_ret_20d", "spx_momentum_3d", "SPY_zscore_60d", "EWG_Germany_vol_20d"], "is_new": true}, {"model_id": "new_h5_GLOBAL_LogisticRegression_N12_t4", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 5, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "IBEX_Spain_ret_20d", "NEE_NextEra_ret_20d", "VVIX_ret_20d", "GD_GeneralDynamics_zscore_60d", "EWL_Switzerland_vol_20d", "BDX_Becton_Dickinson_ret_20d", "PAYX_Paychex_zscore_60d", "HD_ret_20d", "SJM_JM_Smucker_ret_5d", "spx_abs_ret_max_5d"], "is_new": true}, {"model_id": "new_h5_GLOBAL_LogisticRegression_N12_t5", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 5, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "heston_var_ev_h7", "Michigan_Sentiment_ret_20d", "SJM_JM_Smucker_ret_5d", "Brent_Oil_FRED_ret_5d", "DIS_vol_20d", "PLD_Prologis_ret_5d", "SO_SouthernCo_ret_5d", "SPY_zscore_60d", "XLY_Disc_vol_20d", "3M_ret_5d"], "is_new": true}, {"model_id": "new_h5_GLOBAL_LogisticRegression_N12_t6", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 5, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "XOM_ret_1d", "US3M_Rate_vol_20d", "LMT_LockheedMartin_vol_20d", "CLX_Clorox_vol_20d", "XLF_Fin_vol_20d", "BA_ret_1d", "NVDA_vol_20d", "EOG_EOGResources_ret_5d", "HangSeng_HK_ret_5d", "DHR_vol_20d"], "is_new": true}, {"model_id": "new_h5_GLOBAL_LogisticRegression_N12_t7", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 5, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWG_Germany_ret_20d", "ASX_Australia_vol_20d", "NWL_Newell_ret_20d", "Industrial_Production_zscore_60d", "SBUX_vol_20d", "CPB_CampbellSoup_vol_20d", "DIS_vol_20d", "US30Y_Rate_ret_20d", "SJM_JM_Smucker_ret_1d", "LMT_LockheedMartin_vol_20d"], "is_new": true}, {"model_id": "new_h5_GLOBAL_LogisticRegression_N15_t0", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 5, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "SLB_Schlumberger_ret_1d", "AMT_AmericanTower_ret_1d", "hmm_p_stress", "US30Y_Rate_ret_20d", "AMGN_Amgen_ret_1d", "Brent_Oil_FRED_ret_20d", "EWC_Canada_zscore_60d", "ORCL_vol_20d", "BDX_Becton_Dickinson_ret_20d", "HangSeng_HK_vol_20d", "US3M_Rate_vol_20d", "HUM_Humana_ret_5d", "SCHW_Schwab_ret_5d"], "is_new": true}, {"model_id": "new_h5_GLOBAL_LogisticRegression_N15_t1", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 5, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EQIX_Equinix_ret_5d", "Industrial_Production_zscore_60d", "FedFunds_zscore_60d", "TM_Telephone_ret_1d", "EQR_Equity_ret_1d", "PCAR_PaccarInc_ret_5d", "CPB_CampbellSoup_ret_5d", "US30Y_Rate_ret_20d", "VOD_Vodafone_zscore_60d", "US1Y_Rate_ret_5d", "AORD_AUS_zscore_60d", "HUM_Humana_ret_5d", "INTC_ret_5d"], "is_new": true}, {"model_id": "new_h5_GLOBAL_LogisticRegression_N15_t2", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 5, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "US3M_Rate_vol_20d", "Nikkei_Japan_vol_20d", "EFFR_vol_20d", "LOW_Lowes_ret_20d", "US30Y_Rate_ret_20d", "HangSeng_HK_ret_1d", "QQQ_vol_20d", "heston_var_ev_h3", "AMGN_Amgen_ret_1d", "SBUX_ret_5d", "EFFR_ret_1d", "Core_CPI_zscore_60d", "PPL_PPL_ret_1d"], "is_new": true}, {"model_id": "new_h5_GLOBAL_LogisticRegression_N15_t3", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 5, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWM_Malaysia_ret_1d", "AXP_Amex_ret_20d", "FedFunds_zscore_60d", "GD_GeneralDynamics_zscore_60d", "EWL_Switzerland_zscore_60d", "CLX_Clorox_vol_20d", "heston_var_ev_h5", "XLK_Tech_zscore_60d", "PLD_Prologis_ret_5d", "HangSeng_HK_ret_1d", "BDX_Becton_Dickinson_ret_20d", "MS_MorganStanley_ret_5d", "EWY_Korea_ret_20d"], "is_new": true}, {"model_id": "new_h5_GLOBAL_LogisticRegression_N15_t4", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 5, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "3M_ret_5d", "AVB_AvalonBay_zscore_60d", "XOM_ret_1d", "PAYX_Paychex_ret_20d", "HD_ret_5d", "BLK_BlackRock_zscore_60d", "PLD_Prologis_ret_5d", "EWS_Singapore_ret_5d", "EWQ_France_zscore_60d", "US3M_Rate_zscore_60d", "SBUX_vol_20d", "EWL_Switzerland_vol_20d", "IWM_SmallCap_vol_20d"], "is_new": true}, {"model_id": "new_h5_GLOBAL_LogisticRegression_N15_t5", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 5, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "vix_acceleration_1d", "EWH_HongKong_ret_5d", "CLX_Clorox_vol_20d", "T10Y2Y_Spread_ret_5d", "NOC_Northrop_ret_20d", "SO_SouthernCo_ret_5d", "EWQ_France_zscore_60d", "US7Y_Rate_ret_20d", "TED_Spread_vol_20d", "ITT_ITTInc_ret_5d", "US5Y_Rate_ret_5d", "BTI_BritishAmerican_ret_20d", "EXC_Exelon_ret_1d"], "is_new": true}, {"model_id": "new_h5_GLOBAL_LogisticRegression_N15_t6", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 5, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "MSTR_Bitcoin3_ret_1d", "MS_MorganStanley_ret_1d", "EWM_Malaysia_zscore_60d", "US7Y_Rate_ret_20d", "BTI_BritishAmerican_ret_5d", "T_ret_1d", "US5Y_Rate_ret_5d", "AXP_Amex_vol_20d", "AVB_AvalonBay_zscore_60d", "Core_PCE_zscore_60d", "SJM_JM_Smucker_ret_5d", "3M_ret_5d", "Retail_Sales_zscore_60d"], "is_new": true}, {"model_id": "new_h5_GLOBAL_LogisticRegression_N15_t7", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 5, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "DAX_Germany_vol_20d", "GE_ret_1d", "heston_var_ev_h5", "BTI_BritishAmerican_ret_20d", "XLV_Health_zscore_60d", "SBUX_zscore_60d", "EWG_Germany_vol_20d", "HD_ret_1d", "TED_Spread_vol_20d", "MS_MorganStanley_ret_5d", "ASX_Australia_ret_5d", "PAYX_Paychex_zscore_60d", "CLX_Clorox_vol_20d"], "is_new": true}, {"model_id": "new_h5_GLOBAL_LogisticRegression_N20_t0", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 5, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "LUV_SouthwestAir_ret_5d", "AMGN_Amgen_ret_1d", "SBUX_vol_20d", "heston_var_ev_h3", "LOW_Lowes_ret_5d", "MS_MorganStanley_zscore_60d", "SJM_JM_Smucker_ret_5d", "spx_momentum_3d", "Nikkei_Japan_vol_20d", "T_ret_1d", "AMZN_ret_5d", "HUM_Humana_ret_5d", "SBUX_zscore_60d", "SJM_JM_Smucker_ret_1d", "US3Y_Rate_ret_5d", "BA_ret_1d", "BDX_Becton_Dickinson_ret_20d", "CLX_Clorox_vol_20d"], "is_new": true}, {"model_id": "new_h5_GLOBAL_LogisticRegression_N20_t1", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 5, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "TM_Telephone_vol_20d", "PFE_ret_1d", "Brent_Oil_FRED_ret_20d", "US6M_Rate_ret_20d", "CMCSA_ret_1d", "EWY_Korea_ret_20d", "XOM_ret_1d", "ES_Evergy_ret_1d", "BTI_BritishAmerican_ret_20d", "Core_PCE_zscore_60d", "SBUX_vol_20d", "BDX_Becton_Dickinson_ret_20d", "IWM_SmallCap_vol_20d", "HUM_Humana_ret_5d", "MS_MorganStanley_ret_1d", "MSTR_Bitcoin3_ret_1d", "VOD_Vodafone_zscore_60d", "CPB_CampbellSoup_zscore_60d"], "is_new": true}, {"model_id": "new_h5_GLOBAL_LogisticRegression_N20_t2", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 5, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "SLB_Schlumberger_ret_5d", "US6M_Rate_ret_20d", "DHR_ret_1d", "MS_MorganStanley_ret_5d", "VVIX_ret_20d", "GD_GeneralDynamics_zscore_60d", "TED_Spread_vol_20d", "CMCSA_ret_1d", "EWM_Malaysia_ret_1d", "vix_acceleration_1d", "EWY_Korea_ret_20d", "MRK_Merck_zscore_60d", "DAX_Germany_zscore_60d", "VOD_Vodafone_zscore_60d", "LMT_LockheedMartin_vol_20d", "HD_ret_1d", "gjr_condvar_h1", "BDX_Becton_Dickinson_ret_20d"], "is_new": true}, {"model_id": "new_h5_GLOBAL_LogisticRegression_N20_t3", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 5, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "IYM_BasicMaterials_ret_20d", "EWM_Malaysia_zscore_60d", "AORD_AUS_zscore_60d", "IBEX_Spain_ret_20d", "DE_Deere_vol_20d", "CPB_CampbellSoup_zscore_60d", "HD_ret_5d", "EWQ_France_zscore_60d", "CI_Cigna_vol_20d", "NVDA_vol_20d", "NEE_NextEra_ret_20d", "LMT_LockheedMartin_ret_1d", "Core_CPI_zscore_60d", "US1Y_Rate_ret_5d", "VRP_ma5", "VOD_Vodafone_zscore_60d", "XOM_ret_1d", "EXC_Exelon_ret_1d"], "is_new": true}, {"model_id": "new_h5_GLOBAL_LogisticRegression_N20_t4", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 5, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "AXP_Amex_ret_20d", "DHR_ret_1d", "US6M_Rate_ret_20d", "TM_Telephone_vol_20d", "CMCSA_ret_1d", "Nikkei_Japan_zscore_60d", "SLB_Schlumberger_ret_1d", "SBUX_vol_20d", "US1Y_Rate_ret_5d", "heston_var_ev_h3", "EWL_Switzerland_vol_20d", "EWJ_Japan_vol_20d", "ES_Evergy_ret_1d", "MSTR_Bitcoin3_ret_20d", "EFFR_vol_20d", "DAX_Germany_vol_20d", "AMGN_Amgen_ret_1d", "AMT_AmericanTower_ret_1d"], "is_new": true}, {"model_id": "new_h5_GLOBAL_LogisticRegression_N20_t5", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 5, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "DE_Deere_ret_5d", "US3M_Rate_vol_20d", "Core_PCE_zscore_60d", "IYM_BasicMaterials_ret_20d", "MO_AltriaMG_ret_1d", "CPB_CampbellSoup_ret_5d", "ITT_ITTInc_ret_5d", "vix_acceleration_1d", "EOG_EOGResources_vol_20d", "CPB_CampbellSoup_vol_20d", "VOD_Vodafone_zscore_60d", "TED_Spread_zscore_60d", "EWA_Australia_zscore_60d", "NEE_NextEra_ret_20d", "DHR_vol_20d", "EWJ_Japan_vol_20d", "AORD_AUS_zscore_60d", "TM_Telephone_ret_1d"], "is_new": true}, {"model_id": "new_h5_GLOBAL_LogisticRegression_N20_t6", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 5, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "AMGN_Amgen_ret_1d", "SBUX_ret_5d", "Nikkei_Japan_zscore_60d", "CLX_Clorox_vol_20d", "DAX_Germany_zscore_60d", "GILD_Gilead_ret_20d", "AXP_Amex_vol_20d", "IBEX_Spain_ret_20d", "EXC_Exelon_zscore_60d", "Brent_Oil_FRED_ret_5d", "IYR_US_REIT2_zscore_60d", "MS_MorganStanley_ret_5d", "TM_Telephone_vol_20d", "DE_Deere_vol_20d", "hmm_p_stress", "BLK_BlackRock_zscore_60d", "PPL_PPL_ret_1d", "EFFR_ret_1d"], "is_new": true}, {"model_id": "new_h5_GLOBAL_LogisticRegression_N20_t7", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 5, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "Core_PCE_zscore_60d", "SPY_zscore_60d", "XLY_Disc_vol_20d", "T10Y2Y_Spread_ret_5d", "US5Y_Rate_ret_5d", "EOG_EOGResources_vol_20d", "MSTR_Bitcoin3_ret_5d", "NWL_Newell_ret_20d", "EQR_Equity_ret_1d", "EWM_Malaysia_vol_20d", "heston_var_ev_h7", "MSTR_Bitcoin3_ret_20d", "PAYX_Paychex_zscore_60d", "EWY_Korea_zscore_60d", "BDX_Becton_Dickinson_ret_20d", "SLB_Schlumberger_ret_1d", "spx_abs_ret_max_5d", "NOC_Northrop_ret_20d"], "is_new": true}, {"model_id": "new_h5_GLOBAL_LogisticRegression_N25_t0", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 5, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "ENB_EnbridgeInc_ret_1d", "EQIX_Equinix_ret_5d", "US7Y_Rate_ret_20d", "JNJ_ret_1d", "SBUX_zscore_60d", "AMZN_ret_5d", "EWM_Malaysia_ret_1d", "HangSeng_HK_ret_1d", "VRP_ma5", "EWA_Australia_zscore_60d", "LLY_zscore_60d", "MRK_Merck_zscore_60d", "EWJ_Japan_vol_20d", "GE_ret_1d", "Core_PCE_zscore_60d", "QQQ_vol_20d", "MS_MorganStanley_ret_1d", "BLK_BlackRock_zscore_60d", "Retail_Sales_zscore_60d", "T10Y2Y_Spread_ret_5d", "spx_abs_ret_max_5d", "MS_MorganStanley_ret_5d", "XLK_Tech_zscore_60d"], "is_new": true}, {"model_id": "new_h5_GLOBAL_LogisticRegression_N25_t1", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 5, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "WTI_Oil_FRED_zscore_60d", "LOW_Lowes_ret_20d", "ES_Evergy_ret_1d", "AXP_Amex_ret_20d", "VOD_Vodafone_zscore_60d", "TED_Spread_zscore_60d", "HangSeng_HK_vol_20d", "MSTR_Bitcoin3_ret_1d", "EWS_Singapore_ret_5d", "HD_ret_20d", "SBUX_zscore_60d", "PAYX_Paychex_ret_20d", "PLD_Prologis_ret_5d", "EXC_Exelon_ret_1d", "CPB_CampbellSoup_vol_20d", "LLY_zscore_60d", "HangSeng_HK_ret_1d", "Nikkei_Japan_vol_20d", "EWC_Canada_zscore_60d", "EWL_Switzerland_vol_20d", "HangSeng_HK_ret_5d", "XLF_Fin_vol_20d", "DAX_Germany_vol_20d"], "is_new": true}, {"model_id": "new_h5_GLOBAL_LogisticRegression_N25_t2", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 5, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "US1Y_Rate_ret_20d", "AORD_AUS_zscore_60d", "EQIX_Equinix_ret_5d", "US6M_Rate_ret_20d", "BTI_BritishAmerican_ret_20d", "HangSeng_HK_ret_5d", "EWG_Germany_ret_20d", "US5Y_Rate_ret_5d", "EXC_Exelon_zscore_60d", "DIS_vol_20d", "BA_ret_1d", "PAYX_Paychex_ret_20d", "heston_var_ev_h3", "SJM_JM_Smucker_ret_5d", "BDX_Becton_Dickinson_ret_20d", "BTI_BritishAmerican_ret_5d", "IYR_US_REIT2_zscore_60d", "LOW_Lowes_ret_5d", "PFE_ret_1d", "ENB_EnbridgeInc_ret_1d", "XLV_Health_zscore_60d", "LLY_zscore_60d", "MSTR_Bitcoin3_ret_5d"], "is_new": true}, {"model_id": "new_h5_GLOBAL_LogisticRegression_N25_t3", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 5, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "CCI_CrownCastle_vol_20d", "XLB_Materials_zscore_60d", "DHR_ret_1d", "NVDA_vol_20d", "gjr_condvar_h1", "T_ret_1d", "AMGN_Amgen_ret_1d", "PG_ret_20d", "MRK_Merck_zscore_60d", "ASX_Australia_vol_20d", "BA_ret_1d", "US3Y_Rate_ret_5d", "MS_MorganStanley_ret_1d", "PFE_ret_1d", "3M_vol_20d", "SO_SouthernCo_ret_5d", "EWQ_France_ret_20d", "EWG_Germany_vol_20d", "T10Y2Y_Spread_ret_5d", "TED_Spread_zscore_60d", "TED_Spread_vol_20d", "AMD_ret_1d", "Nikkei_Japan_vol_20d"], "is_new": true}, {"model_id": "new_h5_GLOBAL_LogisticRegression_N25_t4", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 5, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "TED_Spread_zscore_60d", "DAX_Germany_zscore_60d", "SLB_Schlumberger_ret_1d", "SBUX_ret_5d", "SLB_Schlumberger_ret_5d", "XLV_Health_zscore_60d", "heston_var_ev_h3", "XLY_Disc_vol_20d", "CCI_CrownCastle_vol_20d", "PAYX_Paychex_ret_20d", "DOW_Price_zscore_60d", "EWC_Canada_zscore_60d", "Industrial_Production_zscore_60d", "EQR_Equity_ret_1d", "US1Y_Rate_ret_5d", "TM_Telephone_ret_1d", "CPB_CampbellSoup_zscore_60d", "LOW_Lowes_ret_20d", "CPB_CampbellSoup_ret_20d", "EWA_Australia_ret_1d", "Nikkei_Japan_zscore_60d", "EOG_EOGResources_ret_5d", "spx_vol_5d"], "is_new": true}, {"model_id": "new_h5_GLOBAL_LogisticRegression_N25_t5", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 5, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EQIX_Equinix_ret_5d", "EWH_HongKong_ret_5d", "US6M_Rate_ret_20d", "PPL_PPL_ret_1d", "PAYX_Paychex_zscore_60d", "AMGN_Amgen_ret_1d", "GD_GeneralDynamics_zscore_60d", "EWL_Switzerland_zscore_60d", "BTI_BritishAmerican_ret_5d", "vix_acceleration_1d", "HD_zscore_60d", "LMT_LockheedMartin_vol_20d", "MO_AltriaMG_ret_1d", "BTI_BritishAmerican_ret_20d", "NEE_NextEra_ret_20d", "EWG_Germany_vol_20d", "TED_Spread_zscore_60d", "SO_SouthernCo_ret_5d", "SLB_Schlumberger_ret_1d", "EWS_Singapore_ret_5d", "vix_mean_abs_ret_5d", "WTI_Oil_FRED_zscore_60d", "PAYX_Paychex_ret_20d"], "is_new": true}, {"model_id": "new_h5_GLOBAL_LogisticRegression_N25_t6", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 5, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "PG_ret_20d", "ITT_ITTInc_ret_5d", "FedFunds_zscore_60d", "XLV_Health_zscore_60d", "TXN_vol_20d", "Brent_Oil_FRED_ret_5d", "Core_CPI_zscore_60d", "hmm_p_stress", "TGT_Target_zscore_60d", "AMD_ret_5d", "spx_momentum_3d", "US30Y_Rate_ret_20d", "CMCSA_ret_1d", "NEE_NextEra_ret_20d", "ORCL_vol_20d", "AMD_ret_1d", "spx_abs_ret_max_5d", "US3M_Rate_zscore_60d", "T_ret_1d", "CPB_CampbellSoup_zscore_60d", "Core_PCE_zscore_60d", "EFFR_ret_1d", "SLB_Schlumberger_ret_5d"], "is_new": true}, {"model_id": "new_h5_GLOBAL_LogisticRegression_N25_t7", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 5, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "GD_GeneralDynamics_zscore_60d", "EWQ_France_ret_20d", "US1Y_Rate_ret_20d", "HUM_Humana_ret_5d", "ASX_Australia_ret_5d", "CPB_CampbellSoup_zscore_60d", "CMCSA_ret_1d", "Industrial_Production_zscore_60d", "gjr_condvar_h1", "HD_ret_20d", "AORD_AUS_zscore_60d", "EWG_Germany_ret_20d", "EWC_Canada_zscore_60d", "JNJ_ret_1d", "XLK_Tech_zscore_60d", "NWL_Newell_ret_20d", "US7Y_Rate_ret_20d", "BDX_Becton_Dickinson_ret_20d", "BLK_BlackRock_zscore_60d", "PFE_ret_1d", "AVB_AvalonBay_zscore_60d", "IYM_BasicMaterials_ret_20d", "INTC_ret_1d"], "is_new": true}, {"model_id": "new_h5_GLOBAL_LogisticRegression_N30_t0", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 5, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "SBUX_zscore_60d", "PAYX_Paychex_vol_20d", "CI_Cigna_vol_20d", "US1Y_Rate_ret_20d", "NVDA_vol_20d", "US7Y_Rate_ret_20d", "SPY_zscore_60d", "NFCI_ret_5d", "TM_Telephone_ret_1d", "EWC_Canada_zscore_60d", "BLK_BlackRock_zscore_60d", "CTAS_Cintas_vol_20d", "PAYX_Paychex_ret_20d", "SCHW_Schwab_ret_5d", "NWL_Newell_ret_20d", "EQIX_Equinix_ret_5d", "Michigan_Sentiment_ret_20d", "BA_ret_1d", "EWY_Korea_zscore_60d", "DIS_vol_20d", "Retail_Sales_zscore_60d", "HangSeng_HK_vol_20d", "MO_AltriaMG_ret_1d", "EWS_Singapore_ret_5d", "PPL_PPL_ret_1d", "MRK_Merck_zscore_60d", "M_Macys_vol_20d", "NOC_Northrop_ret_20d"], "is_new": true}, {"model_id": "new_h5_GLOBAL_LogisticRegression_N30_t1", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 5, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "Michigan_Sentiment_ret_20d", "XLB_Materials_zscore_60d", "DHR_vol_20d", "US30Y_Rate_ret_20d", "AMGN_Amgen_ret_1d", "HangSeng_HK_vol_20d", "Core_PCE_zscore_60d", "PAYX_Paychex_zscore_60d", "XLV_Health_zscore_60d", "gjr_condvar_h1", "MRK_Merck_zscore_60d", "US1Y_Rate_ret_5d", "TM_Telephone_ret_1d", "EWL_Switzerland_vol_20d", "AVB_AvalonBay_zscore_60d", "EWQ_France_ret_20d", "EWM_Malaysia_ret_1d", "JNJ_ret_1d", "BA_ret_1d", "AMZN_ret_5d", "XLY_Disc_vol_20d", "PAYX_Paychex_ret_20d", "MSTR_Bitcoin3_ret_5d", "MS_MorganStanley_ret_5d", "heston_var_ev_h3", "ASX_Australia_ret_5d", "EQR_Equity_ret_1d", "SPY_zscore_60d"], "is_new": true}, {"model_id": "new_h5_GLOBAL_LogisticRegression_N30_t2", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 5, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "US1Y_Rate_ret_5d", "HD_ret_1d", "ORCL_zscore_60d", "CCI_CrownCastle_vol_20d", "3M_vol_20d", "NVDA_vol_20d", "SPY_zscore_60d", "XLY_Disc_vol_20d", "SCHW_Schwab_ret_5d", "EWA_Australia_zscore_60d", "Brent_Oil_FRED_ret_5d", "PLD_Prologis_ret_5d", "EWY_Korea_ret_20d", "Nikkei_Japan_zscore_60d", "PAYX_Paychex_vol_20d", "DOW_Price_zscore_60d", "XOM_ret_1d", "spx_momentum_3d", "BDX_Becton_Dickinson_ret_20d", "Core_PCE_zscore_60d", "AORD_AUS_zscore_60d", "HD_zscore_60d", "LMT_LockheedMartin_vol_20d", "US5Y_Rate_ret_5d", "PG_ret_20d", "US7Y_Rate_ret_20d", "CPB_CampbellSoup_vol_20d", "XLF_Fin_vol_20d"], "is_new": true}, {"model_id": "new_h5_GLOBAL_LogisticRegression_N30_t3", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 5, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "SJM_JM_Smucker_ret_1d", "EWM_Malaysia_ret_1d", "AMT_AmericanTower_ret_1d", "TXN_vol_20d", "Brent_Oil_FRED_ret_5d", "MSTR_Bitcoin3_ret_1d", "EWG_Germany_ret_20d", "HD_ret_5d", "SJM_JM_Smucker_ret_5d", "Core_CPI_zscore_60d", "CI_Cigna_vol_20d", "spx_vol_5d", "Industrial_Production_zscore_60d", "XOM_ret_20d", "DE_Deere_vol_20d", "PG_ret_20d", "ENB_EnbridgeInc_ret_1d", "IYM_BasicMaterials_ret_20d", "ORCL_zscore_60d", "PCAR_PaccarInc_ret_5d", "EOG_EOGResources_vol_20d", "Nikkei_Japan_zscore_60d", "gjr_condvar_h1", "INTC_ret_5d", "TGT_Target_zscore_60d", "IBEX_Spain_ret_20d", "EWS_Singapore_ret_5d", "MS_MorganStanley_ret_1d"], "is_new": true}, {"model_id": "new_h5_GLOBAL_LogisticRegression_N30_t4", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 5, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "LMT_LockheedMartin_ret_1d", "US30Y_Rate_ret_20d", "LOW_Lowes_ret_20d", "TXN_vol_20d", "DIS_vol_20d", "HD_zscore_60d", "EWA_Australia_zscore_60d", "HangSeng_HK_ret_1d", "TED_Spread_vol_20d", "EFFR_ret_1d", "DHR_vol_20d", "heston_var_ev_h3", "3M_vol_20d", "SLB_Schlumberger_ret_5d", "IYM_BasicMaterials_ret_20d", "PAYX_Paychex_vol_20d", "MSTR_Bitcoin3_ret_20d", "EOG_EOGResources_ret_5d", "EWJ_Japan_vol_20d", "MSTR_Bitcoin3_ret_5d", "XLB_Materials_zscore_60d", "EWQ_France_ret_20d", "US5Y_Rate_ret_5d", "CI_Cigna_vol_20d", "NEE_NextEra_ret_20d", "CPB_CampbellSoup_vol_20d", "EWY_Korea_ret_20d", "Industrial_Production_zscore_60d"], "is_new": true}, {"model_id": "new_h5_GLOBAL_LogisticRegression_N30_t5", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 5, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "INTC_ret_5d", "EQIX_Equinix_ret_5d", "Industrial_Production_zscore_60d", "MS_MorganStanley_ret_5d", "SO_SouthernCo_ret_5d", "PAYX_Paychex_vol_20d", "XOM_ret_20d", "BA_ret_1d", "TED_Spread_vol_20d", "TM_Telephone_ret_1d", "AVB_AvalonBay_zscore_60d", "spx_momentum_3d", "EWG_Germany_vol_20d", "LMT_LockheedMartin_ret_1d", "Retail_Sales_zscore_60d", "DHR_vol_20d", "US6M_Rate_ret_20d", "SJM_JM_Smucker_ret_5d", "WTI_Oil_FRED_zscore_60d", "T_ret_1d", "DIS_vol_20d", "AXP_Amex_ret_20d", "LLY_zscore_60d", "HangSeng_HK_vol_20d", "CMCSA_ret_1d", "EWQ_France_zscore_60d", "PLD_Prologis_ret_5d", "EOG_EOGResources_vol_20d"], "is_new": true}, {"model_id": "new_h5_GLOBAL_LogisticRegression_N30_t6", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 5, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "ORCL_zscore_60d", "CI_Cigna_vol_20d", "M_Macys_vol_20d", "HD_ret_20d", "EWM_Malaysia_zscore_60d", "CTAS_Cintas_vol_20d", "XOM_ret_20d", "spx_vol_5d", "XLY_Disc_vol_20d", "IYR_US_REIT2_zscore_60d", "T_ret_1d", "BLK_BlackRock_zscore_60d", "heston_var_ev_h7", "NFCI_ret_5d", "T10Y2Y_Spread_ret_5d", "FedFunds_zscore_60d", "Brent_Oil_FRED_ret_5d", "XLF_Fin_vol_20d", "Nikkei_Japan_vol_20d", "ASX_Australia_ret_5d", "ENB_EnbridgeInc_ret_1d", "BA_ret_1d", "BTI_BritishAmerican_ret_20d", "EOG_EOGResources_ret_5d", "DIS_vol_20d", "CPB_CampbellSoup_zscore_60d", "DHR_ret_1d", "LMT_LockheedMartin_vol_20d"], "is_new": true}, {"model_id": "new_h5_GLOBAL_LogisticRegression_N30_t7", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 5, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWQ_France_ret_20d", "TED_Spread_vol_20d", "PFE_ret_1d", "IBEX_Spain_ret_20d", "SO_SouthernCo_ret_5d", "HD_ret_5d", "SLB_Schlumberger_ret_1d", "US3Y_Rate_ret_5d", "EWM_Malaysia_vol_20d", "NEE_NextEra_ret_20d", "heston_var_ev_h5", "heston_var_ev_h7", "AMD_ret_5d", "DAX_Germany_zscore_60d", "ENB_EnbridgeInc_ret_1d", "Industrial_Production_zscore_60d", "AXP_Amex_ret_20d", "US3M_Rate_vol_20d", "CPB_CampbellSoup_vol_20d", "EWL_Switzerland_vol_20d", "MRK_Merck_zscore_60d", "PLD_Prologis_ret_5d", "US30Y_Rate_ret_20d", "CMCSA_ret_1d", "VVIX_ret_20d", "PPL_PPL_ret_1d", "GD_GeneralDynamics_zscore_60d", "EQR_Equity_ret_1d"], "is_new": true}, {"model_id": "new_h7_CALM_XGBoost_N5_t0", "algo": "XGBoost", "regime": "CALM", "horizon": 7, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "PPL_PPL_ret_1d", "AMT_AmericanTower_ret_1d", "US3M_Rate_zscore_60d"], "is_new": true}, {"model_id": "new_h7_CALM_XGBoost_N5_t1", "algo": "XGBoost", "regime": "CALM", "horizon": 7, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "BLK_BlackRock_zscore_60d", "GD_GeneralDynamics_zscore_60d", "Michigan_Sentiment_ret_20d"], "is_new": true}, {"model_id": "new_h7_CALM_XGBoost_N5_t2", "algo": "XGBoost", "regime": "CALM", "horizon": 7, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "Core_PCE_zscore_60d", "CTAS_Cintas_vol_20d", "AMZN_ret_5d"], "is_new": true}, {"model_id": "new_h7_CALM_XGBoost_N5_t3", "algo": "XGBoost", "regime": "CALM", "horizon": 7, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "vix_mean_abs_ret_5d", "NFCI_ret_5d", "CPB_CampbellSoup_zscore_60d"], "is_new": true}, {"model_id": "new_h7_CALM_XGBoost_N5_t4", "algo": "XGBoost", "regime": "CALM", "horizon": 7, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "HUM_Humana_ret_5d", "AMD_ret_5d", "LLY_zscore_60d"], "is_new": true}, {"model_id": "new_h7_CALM_XGBoost_N5_t5", "algo": "XGBoost", "regime": "CALM", "horizon": 7, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "SJM_JM_Smucker_ret_5d", "CPB_CampbellSoup_ret_5d", "XLB_Materials_zscore_60d"], "is_new": true}, {"model_id": "new_h7_CALM_XGBoost_N5_t6", "algo": "XGBoost", "regime": "CALM", "horizon": 7, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EXC_Exelon_ret_1d", "LOW_Lowes_ret_5d", "EOG_EOGResources_ret_5d"], "is_new": true}, {"model_id": "new_h7_CALM_XGBoost_N5_t7", "algo": "XGBoost", "regime": "CALM", "horizon": 7, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "DAX_Germany_zscore_60d", "US7Y_Rate_ret_20d", "INTC_ret_1d"], "is_new": true}, {"model_id": "new_h7_CALM_XGBoost_N8_t0", "algo": "XGBoost", "regime": "CALM", "horizon": 7, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "DIS_vol_20d", "HD_ret_1d", "BTI_BritishAmerican_ret_5d", "CPB_CampbellSoup_ret_5d", "3M_ret_5d", "SPY_zscore_60d"], "is_new": true}, {"model_id": "new_h7_CALM_XGBoost_N8_t1", "algo": "XGBoost", "regime": "CALM", "horizon": 7, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "AMD_ret_1d", "XLK_Tech_zscore_60d", "US3M_Rate_vol_20d", "EOG_EOGResources_ret_5d", "EOG_EOGResources_vol_20d", "XLY_Disc_vol_20d"], "is_new": true}, {"model_id": "new_h7_CALM_XGBoost_N8_t2", "algo": "XGBoost", "regime": "CALM", "horizon": 7, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "M_Macys_vol_20d", "ENB_EnbridgeInc_ret_1d", "VVIX_ret_20d", "PAYX_Paychex_zscore_60d", "EFFR_vol_20d", "LUV_SouthwestAir_ret_5d"], "is_new": true}, {"model_id": "new_h7_CALM_XGBoost_N8_t3", "algo": "XGBoost", "regime": "CALM", "horizon": 7, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "GILD_Gilead_ret_20d", "MO_AltriaMG_ret_1d", "XLB_Materials_zscore_60d", "VRP_ma5", "INTC_ret_5d", "PCAR_PaccarInc_ret_5d"], "is_new": true}, {"model_id": "new_h7_CALM_XGBoost_N8_t4", "algo": "XGBoost", "regime": "CALM", "horizon": 7, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "SPY_zscore_60d", "MRK_Merck_zscore_60d", "IYR_US_REIT2_zscore_60d", "spx_vol_5d", "AMD_ret_1d", "AXP_Amex_vol_20d"], "is_new": true}, {"model_id": "new_h7_CALM_XGBoost_N8_t5", "algo": "XGBoost", "regime": "CALM", "horizon": 7, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EFFR_vol_20d", "VVIX_ret_20d", "MSTR_Bitcoin3_ret_1d", "PCAR_PaccarInc_ret_5d", "TM_Telephone_ret_1d", "TM_Telephone_vol_20d"], "is_new": true}, {"model_id": "new_h7_CALM_XGBoost_N8_t6", "algo": "XGBoost", "regime": "CALM", "horizon": 7, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "SO_SouthernCo_ret_5d", "IWM_SmallCap_vol_20d", "XOM_ret_1d", "AMZN_ret_5d", "GD_GeneralDynamics_zscore_60d", "AVB_AvalonBay_zscore_60d"], "is_new": true}, {"model_id": "new_h7_CALM_XGBoost_N8_t7", "algo": "XGBoost", "regime": "CALM", "horizon": 7, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWL_Switzerland_zscore_60d", "HD_ret_20d", "Core_CPI_zscore_60d", "AMD_ret_1d", "DOW_Price_zscore_60d", "EWM_Malaysia_zscore_60d"], "is_new": true}, {"model_id": "new_h7_CALM_XGBoost_N10_t0", "algo": "XGBoost", "regime": "CALM", "horizon": 7, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "BA_ret_1d", "CTAS_Cintas_vol_20d", "BTI_BritishAmerican_ret_5d", "EWS_Singapore_ret_5d", "DHR_ret_1d", "MS_MorganStanley_ret_1d", "EWQ_France_zscore_60d", "EWY_Korea_ret_20d"], "is_new": true}, {"model_id": "new_h7_CALM_XGBoost_N10_t1", "algo": "XGBoost", "regime": "CALM", "horizon": 7, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "Nikkei_Japan_vol_20d", "NOC_Northrop_ret_20d", "TED_Spread_zscore_60d", "Michigan_Sentiment_ret_20d", "SBUX_ret_5d", "EWG_Germany_ret_20d", "GILD_Gilead_ret_20d", "SO_SouthernCo_ret_5d"], "is_new": true}, {"model_id": "new_h7_CALM_XGBoost_N10_t2", "algo": "XGBoost", "regime": "CALM", "horizon": 7, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EQR_Equity_ret_1d", "EOG_EOGResources_ret_5d", "CCI_CrownCastle_vol_20d", "WTI_Oil_FRED_zscore_60d", "AMD_ret_5d", "EXC_Exelon_ret_1d", "EWA_Australia_zscore_60d", "spx_momentum_3d"], "is_new": true}, {"model_id": "new_h7_CALM_XGBoost_N10_t3", "algo": "XGBoost", "regime": "CALM", "horizon": 7, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "US3Y_Rate_ret_5d", "BDX_Becton_Dickinson_ret_20d", "GD_GeneralDynamics_zscore_60d", "EWQ_France_ret_20d", "NFCI_ret_5d", "HangSeng_HK_ret_5d", "EXC_Exelon_ret_1d", "spx_vol_5d"], "is_new": true}, {"model_id": "new_h7_CALM_XGBoost_N10_t4", "algo": "XGBoost", "regime": "CALM", "horizon": 7, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "heston_var_ev_h7", "BTI_BritishAmerican_ret_20d", "Brent_Oil_FRED_ret_5d", "AMD_ret_5d", "NFCI_ret_5d", "PAYX_Paychex_ret_20d", "XLV_Health_zscore_60d", "XLK_Tech_zscore_60d"], "is_new": true}, {"model_id": "new_h7_CALM_XGBoost_N10_t5", "algo": "XGBoost", "regime": "CALM", "horizon": 7, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "heston_ev_h3", "ASX_Australia_vol_20d", "EWM_Malaysia_vol_20d", "EOG_EOGResources_ret_5d", "NEE_NextEra_ret_20d", "XOM_ret_20d", "CLX_Clorox_vol_20d", "Industrial_Production_zscore_60d"], "is_new": true}, {"model_id": "new_h7_CALM_XGBoost_N10_t6", "algo": "XGBoost", "regime": "CALM", "horizon": 7, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EXC_Exelon_ret_1d", "AMT_AmericanTower_ret_1d", "HangSeng_HK_vol_20d", "GILD_Gilead_ret_20d", "Nikkei_Japan_vol_20d", "EXC_Exelon_zscore_60d", "M_Macys_vol_20d", "EWM_Malaysia_vol_20d"], "is_new": true}, {"model_id": "new_h7_CALM_XGBoost_N10_t7", "algo": "XGBoost", "regime": "CALM", "horizon": 7, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "ASX_Australia_vol_20d", "BTI_BritishAmerican_ret_20d", "HangSeng_HK_vol_20d", "EXC_Exelon_ret_1d", "gjr_condvar_h1", "XLV_Health_zscore_60d", "SBUX_vol_20d", "US3Y_Rate_ret_5d"], "is_new": true}, {"model_id": "new_h7_CALM_XGBoost_N12_t0", "algo": "XGBoost", "regime": "CALM", "horizon": 7, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "IWM_SmallCap_vol_20d", "SLB_Schlumberger_ret_1d", "BLK_BlackRock_zscore_60d", "IYR_US_REIT2_zscore_60d", "GD_GeneralDynamics_zscore_60d", "CTAS_Cintas_vol_20d", "Nikkei_Japan_zscore_60d", "EWS_Singapore_ret_5d", "CPB_CampbellSoup_vol_20d", "SLB_Schlumberger_ret_5d"], "is_new": true}, {"model_id": "new_h7_CALM_XGBoost_N12_t1", "algo": "XGBoost", "regime": "CALM", "horizon": 7, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "ASX_Australia_vol_20d", "3M_ret_5d", "NFCI_ret_5d", "spx_momentum_3d", "US3M_Rate_vol_20d", "EFFR_vol_20d", "CPB_CampbellSoup_vol_20d", "IYR_US_REIT2_zscore_60d", "EWA_Australia_ret_1d", "XLK_Tech_zscore_60d"], "is_new": true}, {"model_id": "new_h7_CALM_XGBoost_N12_t2", "algo": "XGBoost", "regime": "CALM", "horizon": 7, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "AMZN_ret_5d", "NOC_Northrop_ret_20d", "SBUX_ret_5d", "HD_ret_5d", "SJM_JM_Smucker_ret_5d", "XOM_ret_20d", "DAX_Germany_vol_20d", "PAYX_Paychex_vol_20d", "XLV_Health_zscore_60d", "TM_Telephone_ret_1d"], "is_new": true}, {"model_id": "new_h7_CALM_XGBoost_N12_t3", "algo": "XGBoost", "regime": "CALM", "horizon": 7, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "ASX_Australia_ret_5d", "EWQ_France_zscore_60d", "VRP_ma5", "EWQ_France_ret_20d", "SBUX_ret_5d", "LOW_Lowes_ret_20d", "heston_var_ev_h5", "vix_acceleration_1d", "SLB_Schlumberger_ret_5d", "CPB_CampbellSoup_vol_20d"], "is_new": true}, {"model_id": "new_h7_CALM_XGBoost_N12_t4", "algo": "XGBoost", "regime": "CALM", "horizon": 7, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "CLX_Clorox_vol_20d", "PFE_ret_1d", "AMT_AmericanTower_ret_1d", "XLK_Tech_zscore_60d", "US7Y_Rate_ret_20d", "HD_ret_20d", "VOD_Vodafone_zscore_60d", "ITT_ITTInc_ret_5d", "PG_ret_20d", "BTI_BritishAmerican_ret_5d"], "is_new": true}, {"model_id": "new_h7_CALM_XGBoost_N12_t5", "algo": "XGBoost", "regime": "CALM", "horizon": 7, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "Core_CPI_zscore_60d", "BLK_BlackRock_zscore_60d", "ES_Evergy_ret_1d", "VVIX_ret_20d", "AMZN_ret_5d", "ORCL_vol_20d", "Core_PCE_zscore_60d", "MS_MorganStanley_ret_1d", "US3M_Rate_zscore_60d", "CPB_CampbellSoup_ret_5d"], "is_new": true}, {"model_id": "new_h7_CALM_XGBoost_N12_t6", "algo": "XGBoost", "regime": "CALM", "horizon": 7, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "Brent_Oil_FRED_ret_5d", "BDX_Becton_Dickinson_ret_20d", "NOC_Northrop_ret_20d", "SJM_JM_Smucker_ret_1d", "AORD_AUS_zscore_60d", "PLD_Prologis_ret_5d", "MS_MorganStanley_zscore_60d", "VOD_Vodafone_zscore_60d", "MSTR_Bitcoin3_ret_1d", "SBUX_ret_5d"], "is_new": true}, {"model_id": "new_h7_CALM_XGBoost_N12_t7", "algo": "XGBoost", "regime": "CALM", "horizon": 7, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "DE_Deere_ret_5d", "QQQ_vol_20d", "CPB_CampbellSoup_vol_20d", "PPL_PPL_ret_1d", "SBUX_ret_5d", "EWS_Singapore_ret_5d", "EFFR_vol_20d", "MS_MorganStanley_zscore_60d", "TM_Telephone_ret_1d", "spx_abs_ret_max_5d"], "is_new": true}, {"model_id": "new_h7_CALM_XGBoost_N15_t0", "algo": "XGBoost", "regime": "CALM", "horizon": 7, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "LLY_zscore_60d", "DAX_Germany_zscore_60d", "SJM_JM_Smucker_ret_5d", "VRP_ma5", "HD_zscore_60d", "BTI_BritishAmerican_ret_20d", "NEE_NextEra_ret_20d", "spx_vol_5d", "US1Y_Rate_ret_20d", "NWL_Newell_ret_20d", "EWM_Malaysia_zscore_60d", "Brent_Oil_FRED_ret_20d", "LOW_Lowes_ret_20d"], "is_new": true}, {"model_id": "new_h7_CALM_XGBoost_N15_t1", "algo": "XGBoost", "regime": "CALM", "horizon": 7, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "PFE_ret_1d", "MSTR_Bitcoin3_ret_1d", "SBUX_vol_20d", "TM_Telephone_vol_20d", "ORCL_zscore_60d", "XLK_Tech_zscore_60d", "spx_vol_5d", "3M_ret_5d", "ITT_ITTInc_ret_5d", "HangSeng_HK_ret_1d", "DAX_Germany_vol_20d", "CCI_CrownCastle_vol_20d", "NWL_Newell_ret_20d"], "is_new": true}, {"model_id": "new_h7_CALM_XGBoost_N15_t2", "algo": "XGBoost", "regime": "CALM", "horizon": 7, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "LOW_Lowes_ret_20d", "EWH_HongKong_ret_5d", "HD_ret_5d", "AMT_AmericanTower_ret_1d", "heston_var_ev_h3", "INTC_ret_5d", "DIS_vol_20d", "HangSeng_HK_ret_5d", "DHR_ret_1d", "NWL_Newell_ret_20d", "XLB_Materials_zscore_60d", "XOM_ret_1d", "NFCI_ret_5d"], "is_new": true}, {"model_id": "new_h7_CALM_XGBoost_N15_t3", "algo": "XGBoost", "regime": "CALM", "horizon": 7, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "SPY_zscore_60d", "Nikkei_Japan_vol_20d", "XLV_Health_zscore_60d", "US3Y_Rate_ret_5d", "AMD_ret_1d", "BA_ret_1d", "PLD_Prologis_ret_5d", "US1Y_Rate_ret_5d", "EOG_EOGResources_vol_20d", "SJM_JM_Smucker_ret_5d", "Core_PCE_zscore_60d", "M_Macys_vol_20d", "AMD_ret_5d"], "is_new": true}, {"model_id": "new_h7_CALM_XGBoost_N15_t4", "algo": "XGBoost", "regime": "CALM", "horizon": 7, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "VRP_ma5", "heston_var_ev_h5", "NWL_Newell_ret_20d", "GE_ret_1d", "vix_mean_abs_ret_5d", "AVB_AvalonBay_zscore_60d", "HD_zscore_60d", "US3M_Rate_vol_20d", "SBUX_ret_5d", "EWH_HongKong_ret_5d", "US1Y_Rate_ret_20d", "EWQ_France_ret_20d", "HangSeng_HK_vol_20d"], "is_new": true}, {"model_id": "new_h7_CALM_XGBoost_N15_t5", "algo": "XGBoost", "regime": "CALM", "horizon": 7, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "CPB_CampbellSoup_ret_5d", "Brent_Oil_FRED_ret_20d", "MO_AltriaMG_ret_1d", "LOW_Lowes_ret_20d", "SBUX_vol_20d", "heston_var_ev_h3", "SBUX_ret_5d", "BA_ret_1d", "BTI_BritishAmerican_ret_5d", "TM_Telephone_ret_1d", "EWL_Switzerland_zscore_60d", "AMD_ret_5d", "spx_momentum_3d"], "is_new": true}, {"model_id": "new_h7_CALM_XGBoost_N15_t6", "algo": "XGBoost", "regime": "CALM", "horizon": 7, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWY_Korea_zscore_60d", "LLY_zscore_60d", "MSTR_Bitcoin3_ret_1d", "AMD_ret_5d", "XLY_Disc_vol_20d", "HD_zscore_60d", "EWM_Malaysia_zscore_60d", "DOW_Price_zscore_60d", "AMZN_ret_5d", "Core_PCE_zscore_60d", "EWC_Canada_zscore_60d", "AXP_Amex_ret_20d", "DAX_Germany_zscore_60d"], "is_new": true}, {"model_id": "new_h7_CALM_XGBoost_N15_t7", "algo": "XGBoost", "regime": "CALM", "horizon": 7, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "BA_ret_1d", "EWY_Korea_ret_20d", "MRK_Merck_zscore_60d", "PAYX_Paychex_zscore_60d", "CCI_CrownCastle_vol_20d", "EWM_Malaysia_zscore_60d", "EWC_Canada_zscore_60d", "US30Y_Rate_ret_20d", "EXC_Exelon_zscore_60d", "EWY_Korea_zscore_60d", "SBUX_vol_20d", "EMR_Emerson_ret_20d", "XOM_ret_20d"], "is_new": true}, {"model_id": "new_h7_CALM_XGBoost_N20_t0", "algo": "XGBoost", "regime": "CALM", "horizon": 7, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "VVIX_ret_20d", "MS_MorganStanley_ret_1d", "hmm_p_stress", "INTC_ret_1d", "heston_var_ev_h3", "Retail_Sales_zscore_60d", "EWM_Malaysia_zscore_60d", "MO_AltriaMG_ret_1d", "CI_Cigna_vol_20d", "ITT_ITTInc_ret_5d", "spx_momentum_3d", "HD_ret_5d", "US3M_Rate_zscore_60d", "MSTR_Bitcoin3_ret_20d", "AXP_Amex_vol_20d", "MSTR_Bitcoin3_ret_5d", "Nikkei_Japan_zscore_60d", "BLK_BlackRock_zscore_60d"], "is_new": true}, {"model_id": "new_h7_CALM_XGBoost_N20_t1", "algo": "XGBoost", "regime": "CALM", "horizon": 7, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "SBUX_vol_20d", "CMCSA_ret_1d", "DIS_vol_20d", "GE_ret_1d", "EWH_HongKong_ret_5d", "HD_zscore_60d", "Michigan_Sentiment_ret_20d", "heston_var_ev_h5", "US30Y_Rate_ret_20d", "EWL_Switzerland_zscore_60d", "FedFunds_zscore_60d", "TED_Spread_vol_20d", "IWM_SmallCap_vol_20d", "HUM_Humana_ret_5d", "VRP_ma5", "PCAR_PaccarInc_ret_5d", "JNJ_ret_1d", "CPB_CampbellSoup_zscore_60d"], "is_new": true}, {"model_id": "new_h7_CALM_XGBoost_N20_t2", "algo": "XGBoost", "regime": "CALM", "horizon": 7, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "AMZN_ret_5d", "INTC_ret_1d", "SLB_Schlumberger_ret_5d", "EWM_Malaysia_zscore_60d", "DIS_vol_20d", "NWL_Newell_ret_20d", "heston_var_ev_h5", "US7Y_Rate_ret_20d", "PPL_PPL_ret_1d", "heston_ev_h3", "EWQ_France_ret_20d", "Michigan_Sentiment_ret_20d", "BA_ret_1d", "ORCL_vol_20d", "BLK_BlackRock_zscore_60d", "Brent_Oil_FRED_ret_20d", "ORCL_zscore_60d", "XLF_Fin_vol_20d"], "is_new": true}, {"model_id": "new_h7_CALM_XGBoost_N20_t3", "algo": "XGBoost", "regime": "CALM", "horizon": 7, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "XLV_Health_zscore_60d", "Industrial_Production_zscore_60d", "XOM_ret_1d", "EQIX_Equinix_ret_5d", "SJM_JM_Smucker_ret_1d", "US1Y_Rate_ret_20d", "EWS_Singapore_ret_5d", "EWM_Malaysia_zscore_60d", "TGT_Target_zscore_60d", "IYR_US_REIT2_zscore_60d", "MSTR_Bitcoin3_ret_1d", "US30Y_Rate_ret_20d", "EWM_Malaysia_vol_20d", "FedFunds_zscore_60d", "INTC_ret_5d", "Core_CPI_zscore_60d", "Brent_Oil_FRED_ret_5d", "Nikkei_Japan_vol_20d"], "is_new": true}, {"model_id": "new_h7_CALM_XGBoost_N20_t4", "algo": "XGBoost", "regime": "CALM", "horizon": 7, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "LLY_zscore_60d", "CMCSA_ret_1d", "INTC_ret_1d", "spx_abs_ret_max_5d", "CPB_CampbellSoup_ret_20d", "EWL_Switzerland_vol_20d", "SO_SouthernCo_ret_5d", "DOW_Price_zscore_60d", "EOG_EOGResources_ret_5d", "ENB_EnbridgeInc_ret_1d", "NOC_Northrop_ret_20d", "EFFR_ret_1d", "XLV_Health_zscore_60d", "EWC_Canada_zscore_60d", "MSTR_Bitcoin3_ret_20d", "Industrial_Production_zscore_60d", "SLB_Schlumberger_ret_5d", "T_ret_1d"], "is_new": true}, {"model_id": "new_h7_CALM_XGBoost_N20_t5", "algo": "XGBoost", "regime": "CALM", "horizon": 7, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "IYM_BasicMaterials_ret_20d", "DOW_Price_zscore_60d", "ASX_Australia_ret_5d", "HangSeng_HK_ret_5d", "TXN_vol_20d", "DHR_vol_20d", "INTC_ret_5d", "DIS_vol_20d", "MSTR_Bitcoin3_ret_20d", "PLD_Prologis_ret_5d", "WTI_Oil_FRED_zscore_60d", "Brent_Oil_FRED_ret_5d", "EWA_Australia_zscore_60d", "SLB_Schlumberger_ret_1d", "SJM_JM_Smucker_ret_5d", "MRK_Merck_zscore_60d", "CMCSA_ret_1d", "LMT_LockheedMartin_vol_20d"], "is_new": true}, {"model_id": "new_h7_CALM_XGBoost_N20_t6", "algo": "XGBoost", "regime": "CALM", "horizon": 7, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "XLK_Tech_zscore_60d", "HangSeng_HK_ret_1d", "SO_SouthernCo_ret_5d", "PLD_Prologis_ret_5d", "Brent_Oil_FRED_ret_5d", "TED_Spread_zscore_60d", "DHR_ret_1d", "VVIX_ret_20d", "CCI_CrownCastle_vol_20d", "heston_ev_h3", "TGT_Target_zscore_60d", "3M_vol_20d", "SJM_JM_Smucker_ret_1d", "MS_MorganStanley_ret_5d", "NVDA_vol_20d", "CPB_CampbellSoup_ret_5d", "LOW_Lowes_ret_5d", "EXC_Exelon_ret_1d"], "is_new": true}, {"model_id": "new_h7_CALM_XGBoost_N20_t7", "algo": "XGBoost", "regime": "CALM", "horizon": 7, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "T10Y2Y_Spread_ret_5d", "ORCL_zscore_60d", "BA_ret_1d", "AMZN_ret_5d", "CCI_CrownCastle_vol_20d", "EWL_Switzerland_vol_20d", "NFCI_ret_5d", "INTC_ret_1d", "BTI_BritishAmerican_ret_5d", "DIS_vol_20d", "ES_Evergy_ret_1d", "AMD_ret_1d", "BTI_BritishAmerican_ret_20d", "NVDA_vol_20d", "EWG_Germany_ret_20d", "LOW_Lowes_ret_5d", "PAYX_Paychex_vol_20d", "MS_MorganStanley_ret_5d"], "is_new": true}, {"model_id": "new_h7_CALM_XGBoost_N25_t0", "algo": "XGBoost", "regime": "CALM", "horizon": 7, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "IBEX_Spain_ret_20d", "heston_var_ev_h7", "PAYX_Paychex_vol_20d", "EQR_Equity_ret_1d", "IYM_BasicMaterials_ret_20d", "VRP_ma5", "T_ret_1d", "LOW_Lowes_ret_20d", "MSTR_Bitcoin3_ret_20d", "TXN_vol_20d", "GE_ret_1d", "EQIX_Equinix_ret_5d", "CPB_CampbellSoup_ret_5d", "EOG_EOGResources_ret_5d", "DE_Deere_ret_5d", "SO_SouthernCo_ret_5d", "XLB_Materials_zscore_60d", "TED_Spread_zscore_60d", "INTC_ret_5d", "EWH_HongKong_ret_5d", "ITT_ITTInc_ret_5d", "EWY_Korea_zscore_60d", "CPB_CampbellSoup_vol_20d"], "is_new": true}, {"model_id": "new_h7_CALM_XGBoost_N25_t1", "algo": "XGBoost", "regime": "CALM", "horizon": 7, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "heston_var_ev_h3", "US5Y_Rate_ret_5d", "EWY_Korea_zscore_60d", "EQIX_Equinix_ret_5d", "MS_MorganStanley_zscore_60d", "PG_ret_20d", "EWM_Malaysia_zscore_60d", "M_Macys_vol_20d", "EWA_Australia_zscore_60d", "DOW_Price_zscore_60d", "IWM_SmallCap_vol_20d", "CPB_CampbellSoup_zscore_60d", "LMT_LockheedMartin_vol_20d", "EWQ_France_ret_20d", "CMCSA_ret_1d", "AXP_Amex_ret_20d", "FedFunds_zscore_60d", "EWM_Malaysia_ret_1d", "AMT_AmericanTower_ret_1d", "TM_Telephone_vol_20d", "AORD_AUS_zscore_60d", "HD_zscore_60d", "ITT_ITTInc_ret_5d"], "is_new": true}, {"model_id": "new_h7_CALM_XGBoost_N25_t2", "algo": "XGBoost", "regime": "CALM", "horizon": 7, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EOG_EOGResources_ret_5d", "ITT_ITTInc_ret_5d", "XLK_Tech_zscore_60d", "AXP_Amex_ret_20d", "HUM_Humana_ret_5d", "PAYX_Paychex_vol_20d", "MRK_Merck_zscore_60d", "spx_abs_ret_max_5d", "SBUX_ret_5d", "ORCL_vol_20d", "VVIX_ret_20d", "CLX_Clorox_vol_20d", "PLD_Prologis_ret_5d", "CPB_CampbellSoup_zscore_60d", "EWL_Switzerland_zscore_60d", "LLY_zscore_60d", "US3M_Rate_vol_20d", "vix_acceleration_1d", "BTI_BritishAmerican_ret_5d", "SLB_Schlumberger_ret_1d", "HD_ret_1d", "NFCI_ret_5d", "US6M_Rate_ret_20d"], "is_new": true}, {"model_id": "new_h7_CALM_XGBoost_N25_t3", "algo": "XGBoost", "regime": "CALM", "horizon": 7, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "SLB_Schlumberger_ret_5d", "HangSeng_HK_ret_5d", "QQQ_vol_20d", "CI_Cigna_vol_20d", "DHR_vol_20d", "PAYX_Paychex_zscore_60d", "CTAS_Cintas_vol_20d", "XLK_Tech_zscore_60d", "WTI_Oil_FRED_zscore_60d", "DOW_Price_zscore_60d", "SBUX_zscore_60d", "EOG_EOGResources_ret_5d", "vix_acceleration_1d", "spx_momentum_3d", "XLB_Materials_zscore_60d", "TM_Telephone_vol_20d", "SBUX_vol_20d", "TM_Telephone_ret_1d", "EWG_Germany_ret_20d", "EWQ_France_zscore_60d", "EWM_Malaysia_zscore_60d", "EWJ_Japan_vol_20d", "BDX_Becton_Dickinson_ret_20d"], "is_new": true}, {"model_id": "new_h7_CALM_XGBoost_N25_t4", "algo": "XGBoost", "regime": "CALM", "horizon": 7, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "MS_MorganStanley_zscore_60d", "BTI_BritishAmerican_ret_5d", "EWQ_France_zscore_60d", "LOW_Lowes_ret_20d", "JNJ_ret_1d", "DE_Deere_ret_5d", "DHR_ret_1d", "3M_vol_20d", "EFFR_ret_1d", "hmm_p_stress", "PAYX_Paychex_ret_20d", "XLY_Disc_vol_20d", "ORCL_vol_20d", "CPB_CampbellSoup_ret_5d", "SBUX_zscore_60d", "PCAR_PaccarInc_ret_5d", "AORD_AUS_zscore_60d", "ASX_Australia_ret_5d", "CLX_Clorox_vol_20d", "EOG_EOGResources_vol_20d", "TGT_Target_zscore_60d", "AXP_Amex_ret_20d", "US1Y_Rate_ret_20d"], "is_new": true}, {"model_id": "new_h7_CALM_XGBoost_N25_t5", "algo": "XGBoost", "regime": "CALM", "horizon": 7, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EQR_Equity_ret_1d", "hmm_p_stress", "VRP_ma5", "GE_ret_1d", "TXN_vol_20d", "T_ret_1d", "VVIX_ret_20d", "vix_acceleration_1d", "EWY_Korea_ret_20d", "EWL_Switzerland_zscore_60d", "Core_CPI_zscore_60d", "CMCSA_ret_1d", "US3M_Rate_vol_20d", "SLB_Schlumberger_ret_5d", "US3Y_Rate_ret_5d", "Nikkei_Japan_zscore_60d", "heston_var_ev_h5", "EWS_Singapore_ret_5d", "LOW_Lowes_ret_5d", "HangSeng_HK_ret_5d", "ASX_Australia_ret_5d", "MO_AltriaMG_ret_1d", "DHR_ret_1d"], "is_new": true}, {"model_id": "new_h7_CALM_XGBoost_N25_t6", "algo": "XGBoost", "regime": "CALM", "horizon": 7, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "AMT_AmericanTower_ret_1d", "MSTR_Bitcoin3_ret_5d", "AMZN_ret_5d", "gjr_condvar_h1", "JNJ_ret_1d", "MS_MorganStanley_zscore_60d", "HangSeng_HK_vol_20d", "EWH_HongKong_ret_5d", "DHR_ret_1d", "XLF_Fin_vol_20d", "AMD_ret_1d", "AXP_Amex_vol_20d", "SLB_Schlumberger_ret_5d", "T_ret_1d", "XLK_Tech_zscore_60d", "Industrial_Production_zscore_60d", "spx_momentum_3d", "US5Y_Rate_ret_5d", "BTI_BritishAmerican_ret_5d", "IYR_US_REIT2_zscore_60d", "SJM_JM_Smucker_ret_1d", "EQIX_Equinix_ret_5d", "VRP_ma5"], "is_new": true}, {"model_id": "new_h7_CALM_XGBoost_N25_t7", "algo": "XGBoost", "regime": "CALM", "horizon": 7, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "vix_acceleration_1d", "NFCI_ret_5d", "EOG_EOGResources_ret_5d", "EFFR_ret_1d", "XLB_Materials_zscore_60d", "TM_Telephone_ret_1d", "EQIX_Equinix_ret_5d", "EMR_Emerson_ret_20d", "IYM_BasicMaterials_ret_20d", "AMD_ret_5d", "LLY_zscore_60d", "EWS_Singapore_ret_5d", "SLB_Schlumberger_ret_5d", "MS_MorganStanley_ret_5d", "INTC_ret_5d", "EWA_Australia_ret_1d", "CMCSA_ret_1d", "DE_Deere_ret_5d", "EWC_Canada_zscore_60d", "INTC_ret_1d", "US30Y_Rate_ret_20d", "GD_GeneralDynamics_zscore_60d", "SPY_zscore_60d"], "is_new": true}, {"model_id": "new_h7_CALM_XGBoost_N30_t0", "algo": "XGBoost", "regime": "CALM", "horizon": 7, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "US3Y_Rate_ret_5d", "FedFunds_zscore_60d", "CMCSA_ret_1d", "TED_Spread_vol_20d", "GE_ret_1d", "HUM_Humana_ret_5d", "GILD_Gilead_ret_20d", "Nikkei_Japan_zscore_60d", "EFFR_vol_20d", "AXP_Amex_ret_20d", "GD_GeneralDynamics_zscore_60d", "ES_Evergy_ret_1d", "HangSeng_HK_vol_20d", "Industrial_Production_zscore_60d", "SJM_JM_Smucker_ret_1d", "US7Y_Rate_ret_20d", "PLD_Prologis_ret_5d", "XOM_ret_20d", "PAYX_Paychex_vol_20d", "EWA_Australia_zscore_60d", "CLX_Clorox_vol_20d", "EWS_Singapore_ret_5d", "EMR_Emerson_ret_20d", "VRP_ma5", "CPB_CampbellSoup_zscore_60d", "SLB_Schlumberger_ret_5d", "ASX_Australia_ret_5d", "vix_mean_abs_ret_5d"], "is_new": true}, {"model_id": "new_h7_CALM_XGBoost_N30_t1", "algo": "XGBoost", "regime": "CALM", "horizon": 7, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "PFE_ret_1d", "GILD_Gilead_ret_20d", "EWJ_Japan_vol_20d", "TED_Spread_zscore_60d", "JNJ_ret_1d", "heston_var_ev_h7", "vix_mean_abs_ret_5d", "EOG_EOGResources_vol_20d", "XLF_Fin_vol_20d", "AXP_Amex_ret_20d", "EWM_Malaysia_zscore_60d", "DE_Deere_vol_20d", "CI_Cigna_vol_20d", "Nikkei_Japan_zscore_60d", "IYR_US_REIT2_zscore_60d", "DAX_Germany_vol_20d", "INTC_ret_5d", "LUV_SouthwestAir_ret_5d", "Nikkei_Japan_vol_20d", "PCAR_PaccarInc_ret_5d", "EWQ_France_ret_20d", "EWA_Australia_ret_1d", "ORCL_zscore_60d", "MSTR_Bitcoin3_ret_1d", "Brent_Oil_FRED_ret_20d", "AXP_Amex_vol_20d", "VVIX_ret_20d", "DHR_ret_1d"], "is_new": true}, {"model_id": "new_h7_CALM_XGBoost_N30_t2", "algo": "XGBoost", "regime": "CALM", "horizon": 7, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "gjr_condvar_h1", "IYM_BasicMaterials_ret_20d", "heston_var_ev_h5", "XLF_Fin_vol_20d", "SBUX_zscore_60d", "EWQ_France_ret_20d", "DOW_Price_zscore_60d", "HD_zscore_60d", "NVDA_vol_20d", "AMZN_ret_5d", "MO_AltriaMG_ret_1d", "QQQ_vol_20d", "ASX_Australia_vol_20d", "LMT_LockheedMartin_ret_1d", "AMGN_Amgen_ret_1d", "DE_Deere_vol_20d", "CMCSA_ret_1d", "CPB_CampbellSoup_ret_20d", "WTI_Oil_FRED_zscore_60d", "DHR_ret_1d", "vix_acceleration_1d", "AMD_ret_1d", "vix_mean_abs_ret_5d", "SLB_Schlumberger_ret_5d", "EWL_Switzerland_vol_20d", "MS_MorganStanley_zscore_60d", "T10Y2Y_Spread_ret_5d", "EOG_EOGResources_ret_5d"], "is_new": true}, {"model_id": "new_h7_CALM_XGBoost_N30_t3", "algo": "XGBoost", "regime": "CALM", "horizon": 7, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "heston_var_ev_h3", "DIS_vol_20d", "HD_ret_1d", "hmm_p_stress", "LMT_LockheedMartin_vol_20d", "MO_AltriaMG_ret_1d", "vix_acceleration_1d", "MSTR_Bitcoin3_ret_1d", "PCAR_PaccarInc_ret_5d", "EXC_Exelon_ret_1d", "ORCL_vol_20d", "US3M_Rate_zscore_60d", "EWQ_France_ret_20d", "HD_ret_5d", "HangSeng_HK_ret_5d", "EWA_Australia_ret_1d", "EQIX_Equinix_ret_5d", "DE_Deere_ret_5d", "EMR_Emerson_ret_20d", "US1Y_Rate_ret_20d", "EWA_Australia_zscore_60d", "PAYX_Paychex_vol_20d", "EOG_EOGResources_ret_5d", "Brent_Oil_FRED_ret_5d", "EWC_Canada_zscore_60d", "AMD_ret_1d", "ES_Evergy_ret_1d", "SCHW_Schwab_ret_5d"], "is_new": true}, {"model_id": "new_h7_CALM_XGBoost_N30_t4", "algo": "XGBoost", "regime": "CALM", "horizon": 7, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "HangSeng_HK_ret_5d", "vix_mean_abs_ret_5d", "T_ret_1d", "HD_ret_5d", "AMD_ret_1d", "heston_var_ev_h5", "AMZN_ret_5d", "AMD_ret_5d", "GE_ret_1d", "LMT_LockheedMartin_vol_20d", "EWL_Switzerland_vol_20d", "HangSeng_HK_ret_1d", "spx_momentum_3d", "GILD_Gilead_ret_20d", "MS_MorganStanley_ret_5d", "ES_Evergy_ret_1d", "EOG_EOGResources_ret_5d", "SJM_JM_Smucker_ret_1d", "DE_Deere_vol_20d", "SJM_JM_Smucker_ret_5d", "Brent_Oil_FRED_ret_20d", "SCHW_Schwab_ret_5d", "EFFR_ret_1d", "SBUX_ret_5d", "EQIX_Equinix_ret_5d", "XLY_Disc_vol_20d", "EWS_Singapore_ret_5d", "EWC_Canada_zscore_60d"], "is_new": true}, {"model_id": "new_h7_CALM_XGBoost_N30_t5", "algo": "XGBoost", "regime": "CALM", "horizon": 7, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "AMT_AmericanTower_ret_1d", "FedFunds_zscore_60d", "TGT_Target_zscore_60d", "vix_acceleration_1d", "HD_ret_5d", "EWQ_France_zscore_60d", "SBUX_zscore_60d", "MSTR_Bitcoin3_ret_5d", "DIS_vol_20d", "US1Y_Rate_ret_20d", "DOW_Price_zscore_60d", "Industrial_Production_zscore_60d", "MO_AltriaMG_ret_1d", "US3Y_Rate_ret_5d", "XLB_Materials_zscore_60d", "HangSeng_HK_vol_20d", "US1Y_Rate_ret_5d", "spx_abs_ret_max_5d", "US30Y_Rate_ret_20d", "EMR_Emerson_ret_20d", "SJM_JM_Smucker_ret_1d", "LMT_LockheedMartin_ret_1d", "US5Y_Rate_ret_5d", "MS_MorganStanley_ret_1d", "LUV_SouthwestAir_ret_5d", "XOM_ret_1d", "heston_ev_h3", "SCHW_Schwab_ret_5d"], "is_new": true}, {"model_id": "new_h7_CALM_XGBoost_N30_t6", "algo": "XGBoost", "regime": "CALM", "horizon": 7, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "AMZN_ret_5d", "3M_ret_5d", "SLB_Schlumberger_ret_1d", "ORCL_zscore_60d", "CMCSA_ret_1d", "BA_ret_1d", "NWL_Newell_ret_20d", "DIS_vol_20d", "EWL_Switzerland_zscore_60d", "IYR_US_REIT2_zscore_60d", "XLF_Fin_vol_20d", "CTAS_Cintas_vol_20d", "MS_MorganStanley_ret_5d", "US3M_Rate_vol_20d", "EWJ_Japan_vol_20d", "CPB_CampbellSoup_zscore_60d", "spx_vol_5d", "ES_Evergy_ret_1d", "FedFunds_zscore_60d", "GILD_Gilead_ret_20d", "MSTR_Bitcoin3_ret_5d", "HD_ret_5d", "US6M_Rate_ret_20d", "ITT_ITTInc_ret_5d", "M_Macys_vol_20d", "VRP_ma5", "heston_ev_h3", "US5Y_Rate_ret_5d"], "is_new": true}, {"model_id": "new_h7_CALM_XGBoost_N30_t7", "algo": "XGBoost", "regime": "CALM", "horizon": 7, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "vix_acceleration_1d", "AORD_AUS_zscore_60d", "SCHW_Schwab_ret_5d", "Nikkei_Japan_vol_20d", "heston_var_ev_h7", "DHR_vol_20d", "HD_ret_1d", "INTC_ret_1d", "CCI_CrownCastle_vol_20d", "EWQ_France_ret_20d", "BLK_BlackRock_zscore_60d", "US6M_Rate_ret_20d", "NEE_NextEra_ret_20d", "CTAS_Cintas_vol_20d", "hmm_p_stress", "PFE_ret_1d", "EWM_Malaysia_ret_1d", "HD_zscore_60d", "T10Y2Y_Spread_ret_5d", "IYR_US_REIT2_zscore_60d", "SBUX_ret_5d", "IBEX_Spain_ret_20d", "HangSeng_HK_ret_1d", "Brent_Oil_FRED_ret_5d", "US1Y_Rate_ret_5d", "HD_ret_5d", "US7Y_Rate_ret_20d", "ENB_EnbridgeInc_ret_1d"], "is_new": true}, {"model_id": "new_h7_CALM_LightGBM_N5_t0", "algo": "LightGBM", "regime": "CALM", "horizon": 7, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWM_Malaysia_ret_1d", "ORCL_zscore_60d", "AVB_AvalonBay_zscore_60d"], "is_new": true}, {"model_id": "new_h7_CALM_LightGBM_N5_t1", "algo": "LightGBM", "regime": "CALM", "horizon": 7, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "XLY_Disc_vol_20d", "HUM_Humana_ret_5d", "CI_Cigna_vol_20d"], "is_new": true}, {"model_id": "new_h7_CALM_LightGBM_N5_t2", "algo": "LightGBM", "regime": "CALM", "horizon": 7, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "VVIX_ret_20d", "DAX_Germany_vol_20d", "JNJ_ret_1d"], "is_new": true}, {"model_id": "new_h7_CALM_LightGBM_N5_t3", "algo": "LightGBM", "regime": "CALM", "horizon": 7, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EFFR_vol_20d", "VVIX_ret_20d", "IYM_BasicMaterials_ret_20d"], "is_new": true}, {"model_id": "new_h7_CALM_LightGBM_N5_t4", "algo": "LightGBM", "regime": "CALM", "horizon": 7, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "SO_SouthernCo_ret_5d", "CI_Cigna_vol_20d", "Brent_Oil_FRED_ret_5d"], "is_new": true}, {"model_id": "new_h7_CALM_LightGBM_N5_t5", "algo": "LightGBM", "regime": "CALM", "horizon": 7, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "T10Y2Y_Spread_ret_5d", "XLF_Fin_vol_20d", "CTAS_Cintas_vol_20d"], "is_new": true}, {"model_id": "new_h7_CALM_LightGBM_N5_t6", "algo": "LightGBM", "regime": "CALM", "horizon": 7, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "XLB_Materials_zscore_60d", "US3M_Rate_vol_20d", "Core_PCE_zscore_60d"], "is_new": true}, {"model_id": "new_h7_CALM_LightGBM_N5_t7", "algo": "LightGBM", "regime": "CALM", "horizon": 7, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "BLK_BlackRock_zscore_60d", "HangSeng_HK_ret_5d", "SLB_Schlumberger_ret_1d"], "is_new": true}, {"model_id": "new_h7_CALM_LightGBM_N8_t0", "algo": "LightGBM", "regime": "CALM", "horizon": 7, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EFFR_ret_1d", "MSTR_Bitcoin3_ret_1d", "SJM_JM_Smucker_ret_1d", "SO_SouthernCo_ret_5d", "DHR_vol_20d", "EMR_Emerson_ret_20d"], "is_new": true}, {"model_id": "new_h7_CALM_LightGBM_N8_t1", "algo": "LightGBM", "regime": "CALM", "horizon": 7, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "DE_Deere_ret_5d", "SJM_JM_Smucker_ret_1d", "GD_GeneralDynamics_zscore_60d", "XLB_Materials_zscore_60d", "PAYX_Paychex_zscore_60d", "SLB_Schlumberger_ret_1d"], "is_new": true}, {"model_id": "new_h7_CALM_LightGBM_N8_t2", "algo": "LightGBM", "regime": "CALM", "horizon": 7, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "ENB_EnbridgeInc_ret_1d", "SLB_Schlumberger_ret_1d", "BTI_BritishAmerican_ret_20d", "MO_AltriaMG_ret_1d", "Industrial_Production_zscore_60d", "Brent_Oil_FRED_ret_5d"], "is_new": true}, {"model_id": "new_h7_CALM_LightGBM_N8_t3", "algo": "LightGBM", "regime": "CALM", "horizon": 7, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "XLV_Health_zscore_60d", "EWM_Malaysia_zscore_60d", "SBUX_zscore_60d", "DHR_ret_1d", "VRP_ma5", "WTI_Oil_FRED_zscore_60d"], "is_new": true}, {"model_id": "new_h7_CALM_LightGBM_N8_t4", "algo": "LightGBM", "regime": "CALM", "horizon": 7, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWG_Germany_vol_20d", "ASX_Australia_ret_5d", "CPB_CampbellSoup_ret_20d", "Michigan_Sentiment_ret_20d", "BDX_Becton_Dickinson_ret_20d", "HD_ret_5d"], "is_new": true}, {"model_id": "new_h7_CALM_LightGBM_N8_t5", "algo": "LightGBM", "regime": "CALM", "horizon": 7, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "MO_AltriaMG_ret_1d", "XOM_ret_1d", "WTI_Oil_FRED_zscore_60d", "heston_var_ev_h3", "Retail_Sales_zscore_60d", "EQR_Equity_ret_1d"], "is_new": true}, {"model_id": "new_h7_CALM_LightGBM_N8_t6", "algo": "LightGBM", "regime": "CALM", "horizon": 7, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "SLB_Schlumberger_ret_5d", "JNJ_ret_1d", "PFE_ret_1d", "NFCI_ret_5d", "AORD_AUS_zscore_60d", "MO_AltriaMG_ret_1d"], "is_new": true}, {"model_id": "new_h7_CALM_LightGBM_N8_t7", "algo": "LightGBM", "regime": "CALM", "horizon": 7, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "VVIX_ret_20d", "US5Y_Rate_ret_5d", "PPL_PPL_ret_1d", "LOW_Lowes_ret_5d", "EWC_Canada_zscore_60d", "XLY_Disc_vol_20d"], "is_new": true}, {"model_id": "new_h7_CALM_LightGBM_N10_t0", "algo": "LightGBM", "regime": "CALM", "horizon": 7, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "AVB_AvalonBay_zscore_60d", "LLY_zscore_60d", "HD_ret_1d", "SJM_JM_Smucker_ret_5d", "EWY_Korea_zscore_60d", "NFCI_ret_5d", "LOW_Lowes_ret_20d", "EOG_EOGResources_vol_20d"], "is_new": true}, {"model_id": "new_h7_CALM_LightGBM_N10_t1", "algo": "LightGBM", "regime": "CALM", "horizon": 7, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "T10Y2Y_Spread_ret_5d", "BA_ret_1d", "LUV_SouthwestAir_ret_5d", "TXN_vol_20d", "EWC_Canada_zscore_60d", "AXP_Amex_vol_20d", "PAYX_Paychex_ret_20d", "MSTR_Bitcoin3_ret_5d"], "is_new": true}, {"model_id": "new_h7_CALM_LightGBM_N10_t2", "algo": "LightGBM", "regime": "CALM", "horizon": 7, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWY_Korea_zscore_60d", "DE_Deere_vol_20d", "M_Macys_vol_20d", "AMZN_ret_5d", "DHR_vol_20d", "HD_ret_1d", "GILD_Gilead_ret_20d", "DAX_Germany_zscore_60d"], "is_new": true}, {"model_id": "new_h7_CALM_LightGBM_N10_t3", "algo": "LightGBM", "regime": "CALM", "horizon": 7, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWM_Malaysia_vol_20d", "SJM_JM_Smucker_ret_5d", "HD_ret_20d", "XLV_Health_zscore_60d", "MSTR_Bitcoin3_ret_5d", "EFFR_ret_1d", "PAYX_Paychex_zscore_60d", "FedFunds_zscore_60d"], "is_new": true}, {"model_id": "new_h7_CALM_LightGBM_N10_t4", "algo": "LightGBM", "regime": "CALM", "horizon": 7, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "ASX_Australia_ret_5d", "PAYX_Paychex_ret_20d", "AMT_AmericanTower_ret_1d", "BTI_BritishAmerican_ret_5d", "EWQ_France_zscore_60d", "HangSeng_HK_ret_5d", "DHR_ret_1d", "EOG_EOGResources_vol_20d"], "is_new": true}, {"model_id": "new_h7_CALM_LightGBM_N10_t5", "algo": "LightGBM", "regime": "CALM", "horizon": 7, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "GILD_Gilead_ret_20d", "HangSeng_HK_vol_20d", "AMZN_ret_5d", "MRK_Merck_zscore_60d", "SJM_JM_Smucker_ret_1d", "CTAS_Cintas_vol_20d", "BA_ret_1d", "spx_vol_5d"], "is_new": true}, {"model_id": "new_h7_CALM_LightGBM_N10_t6", "algo": "LightGBM", "regime": "CALM", "horizon": 7, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EFFR_ret_1d", "3M_ret_5d", "Core_CPI_zscore_60d", "hmm_p_stress", "CPB_CampbellSoup_vol_20d", "BA_ret_1d", "EXC_Exelon_ret_1d", "CLX_Clorox_vol_20d"], "is_new": true}, {"model_id": "new_h7_CALM_LightGBM_N10_t7", "algo": "LightGBM", "regime": "CALM", "horizon": 7, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "HD_zscore_60d", "VVIX_ret_20d", "spx_abs_ret_max_5d", "MS_MorganStanley_ret_5d", "3M_ret_5d", "DE_Deere_vol_20d", "heston_var_ev_h5", "EMR_Emerson_ret_20d"], "is_new": true}, {"model_id": "new_h7_CALM_LightGBM_N12_t0", "algo": "LightGBM", "regime": "CALM", "horizon": 7, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "DAX_Germany_vol_20d", "CPB_CampbellSoup_vol_20d", "XLV_Health_zscore_60d", "TM_Telephone_vol_20d", "SJM_JM_Smucker_ret_5d", "IWM_SmallCap_vol_20d", "Industrial_Production_zscore_60d", "EMR_Emerson_ret_20d", "MS_MorganStanley_zscore_60d", "CI_Cigna_vol_20d"], "is_new": true}, {"model_id": "new_h7_CALM_LightGBM_N12_t1", "algo": "LightGBM", "regime": "CALM", "horizon": 7, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "AVB_AvalonBay_zscore_60d", "DAX_Germany_zscore_60d", "MSTR_Bitcoin3_ret_1d", "Core_PCE_zscore_60d", "DE_Deere_ret_5d", "MRK_Merck_zscore_60d", "US3M_Rate_vol_20d", "ORCL_zscore_60d", "US6M_Rate_ret_20d", "SBUX_zscore_60d"], "is_new": true}, {"model_id": "new_h7_CALM_LightGBM_N12_t2", "algo": "LightGBM", "regime": "CALM", "horizon": 7, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EQR_Equity_ret_1d", "PAYX_Paychex_zscore_60d", "CTAS_Cintas_vol_20d", "TGT_Target_zscore_60d", "EWJ_Japan_vol_20d", "AMD_ret_5d", "Core_CPI_zscore_60d", "AMD_ret_1d", "AXP_Amex_vol_20d", "SO_SouthernCo_ret_5d"], "is_new": true}, {"model_id": "new_h7_CALM_LightGBM_N12_t3", "algo": "LightGBM", "regime": "CALM", "horizon": 7, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "JNJ_ret_1d", "EXC_Exelon_zscore_60d", "ASX_Australia_vol_20d", "AMD_ret_1d", "EWJ_Japan_vol_20d", "LUV_SouthwestAir_ret_5d", "vix_mean_abs_ret_5d", "CTAS_Cintas_vol_20d", "spx_vol_5d", "US1Y_Rate_ret_20d"], "is_new": true}, {"model_id": "new_h7_CALM_LightGBM_N12_t4", "algo": "LightGBM", "regime": "CALM", "horizon": 7, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "SCHW_Schwab_ret_5d", "MSTR_Bitcoin3_ret_5d", "HD_ret_20d", "US3M_Rate_vol_20d", "HangSeng_HK_ret_1d", "PAYX_Paychex_ret_20d", "SLB_Schlumberger_ret_1d", "spx_vol_5d", "VRP_ma5", "PCAR_PaccarInc_ret_5d"], "is_new": true}, {"model_id": "new_h7_CALM_LightGBM_N12_t5", "algo": "LightGBM", "regime": "CALM", "horizon": 7, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "CI_Cigna_vol_20d", "Michigan_Sentiment_ret_20d", "hmm_p_stress", "CCI_CrownCastle_vol_20d", "NFCI_ret_5d", "vix_acceleration_1d", "EXC_Exelon_zscore_60d", "MS_MorganStanley_ret_1d", "MSTR_Bitcoin3_ret_20d", "MS_MorganStanley_ret_5d"], "is_new": true}, {"model_id": "new_h7_CALM_LightGBM_N12_t6", "algo": "LightGBM", "regime": "CALM", "horizon": 7, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "Nikkei_Japan_zscore_60d", "EWM_Malaysia_ret_1d", "vix_acceleration_1d", "CCI_CrownCastle_vol_20d", "SO_SouthernCo_ret_5d", "TM_Telephone_vol_20d", "DE_Deere_vol_20d", "VVIX_ret_20d", "ORCL_zscore_60d", "SJM_JM_Smucker_ret_5d"], "is_new": true}, {"model_id": "new_h7_CALM_LightGBM_N12_t7", "algo": "LightGBM", "regime": "CALM", "horizon": 7, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "HD_zscore_60d", "MO_AltriaMG_ret_1d", "T10Y2Y_Spread_ret_5d", "heston_var_ev_h3", "HangSeng_HK_ret_5d", "SLB_Schlumberger_ret_1d", "INTC_ret_5d", "CLX_Clorox_vol_20d", "MS_MorganStanley_ret_5d", "Nikkei_Japan_zscore_60d"], "is_new": true}, {"model_id": "new_h7_CALM_LightGBM_N15_t0", "algo": "LightGBM", "regime": "CALM", "horizon": 7, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "ORCL_zscore_60d", "ITT_ITTInc_ret_5d", "HUM_Humana_ret_5d", "SLB_Schlumberger_ret_5d", "IYM_BasicMaterials_ret_20d", "EOG_EOGResources_vol_20d", "HangSeng_HK_ret_1d", "AORD_AUS_zscore_60d", "LLY_zscore_60d", "GD_GeneralDynamics_zscore_60d", "CTAS_Cintas_vol_20d", "NFCI_ret_5d", "TM_Telephone_ret_1d"], "is_new": true}, {"model_id": "new_h7_CALM_LightGBM_N15_t1", "algo": "LightGBM", "regime": "CALM", "horizon": 7, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EQIX_Equinix_ret_5d", "Core_CPI_zscore_60d", "vix_mean_abs_ret_5d", "HangSeng_HK_vol_20d", "PLD_Prologis_ret_5d", "spx_abs_ret_max_5d", "EWQ_France_ret_20d", "SPY_zscore_60d", "ASX_Australia_ret_5d", "EWA_Australia_ret_1d", "EFFR_vol_20d", "PAYX_Paychex_zscore_60d", "AXP_Amex_vol_20d"], "is_new": true}, {"model_id": "new_h7_CALM_LightGBM_N15_t2", "algo": "LightGBM", "regime": "CALM", "horizon": 7, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "AMD_ret_1d", "EOG_EOGResources_ret_5d", "SBUX_vol_20d", "HUM_Humana_ret_5d", "CMCSA_ret_1d", "EOG_EOGResources_vol_20d", "EWY_Korea_zscore_60d", "EWM_Malaysia_ret_1d", "EFFR_ret_1d", "3M_vol_20d", "IYM_BasicMaterials_ret_20d", "TED_Spread_zscore_60d", "VRP_ma5"], "is_new": true}, {"model_id": "new_h7_CALM_LightGBM_N15_t3", "algo": "LightGBM", "regime": "CALM", "horizon": 7, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "XLK_Tech_zscore_60d", "AMD_ret_5d", "heston_var_ev_h7", "EXC_Exelon_zscore_60d", "hmm_p_stress", "DOW_Price_zscore_60d", "MSTR_Bitcoin3_ret_5d", "ORCL_vol_20d", "CPB_CampbellSoup_ret_20d", "EWS_Singapore_ret_5d", "MSTR_Bitcoin3_ret_20d", "PLD_Prologis_ret_5d", "NOC_Northrop_ret_20d"], "is_new": true}, {"model_id": "new_h7_CALM_LightGBM_N15_t4", "algo": "LightGBM", "regime": "CALM", "horizon": 7, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "TM_Telephone_ret_1d", "IWM_SmallCap_vol_20d", "DIS_vol_20d", "XLF_Fin_vol_20d", "INTC_ret_1d", "SO_SouthernCo_ret_5d", "3M_vol_20d", "EWH_HongKong_ret_5d", "CPB_CampbellSoup_ret_20d", "DOW_Price_zscore_60d", "AMZN_ret_5d", "CTAS_Cintas_vol_20d", "EWM_Malaysia_vol_20d"], "is_new": true}, {"model_id": "new_h7_CALM_LightGBM_N15_t5", "algo": "LightGBM", "regime": "CALM", "horizon": 7, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "SO_SouthernCo_ret_5d", "Industrial_Production_zscore_60d", "MO_AltriaMG_ret_1d", "US3M_Rate_zscore_60d", "Michigan_Sentiment_ret_20d", "ITT_ITTInc_ret_5d", "gjr_condvar_h1", "EFFR_vol_20d", "Brent_Oil_FRED_ret_5d", "CLX_Clorox_vol_20d", "QQQ_vol_20d", "SBUX_zscore_60d", "US1Y_Rate_ret_20d"], "is_new": true}, {"model_id": "new_h7_CALM_LightGBM_N15_t6", "algo": "LightGBM", "regime": "CALM", "horizon": 7, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "BDX_Becton_Dickinson_ret_20d", "INTC_ret_5d", "US3M_Rate_vol_20d", "EWJ_Japan_vol_20d", "XLB_Materials_zscore_60d", "AMT_AmericanTower_ret_1d", "EWS_Singapore_ret_5d", "BA_ret_1d", "MSTR_Bitcoin3_ret_1d", "IBEX_Spain_ret_20d", "DAX_Germany_vol_20d", "SJM_JM_Smucker_ret_1d", "vix_acceleration_1d"], "is_new": true}, {"model_id": "new_h7_CALM_LightGBM_N15_t7", "algo": "LightGBM", "regime": "CALM", "horizon": 7, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "US3M_Rate_vol_20d", "ES_Evergy_ret_1d", "XOM_ret_20d", "XLY_Disc_vol_20d", "BDX_Becton_Dickinson_ret_20d", "PLD_Prologis_ret_5d", "SO_SouthernCo_ret_5d", "EWA_Australia_ret_1d", "GILD_Gilead_ret_20d", "3M_ret_5d", "CPB_CampbellSoup_ret_5d", "AMT_AmericanTower_ret_1d", "PG_ret_20d"], "is_new": true}, {"model_id": "new_h7_CALM_LightGBM_N20_t0", "algo": "LightGBM", "regime": "CALM", "horizon": 7, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "DE_Deere_ret_5d", "DAX_Germany_vol_20d", "AXP_Amex_ret_20d", "US7Y_Rate_ret_20d", "TXN_vol_20d", "Industrial_Production_zscore_60d", "EWL_Switzerland_vol_20d", "BDX_Becton_Dickinson_ret_20d", "heston_ev_h3", "Nikkei_Japan_zscore_60d", "3M_ret_5d", "SLB_Schlumberger_ret_5d", "GE_ret_1d", "EWC_Canada_zscore_60d", "DIS_vol_20d", "XLY_Disc_vol_20d", "CMCSA_ret_1d", "EWS_Singapore_ret_5d"], "is_new": true}, {"model_id": "new_h7_CALM_LightGBM_N20_t1", "algo": "LightGBM", "regime": "CALM", "horizon": 7, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "hmm_p_stress", "Core_PCE_zscore_60d", "SO_SouthernCo_ret_5d", "SBUX_ret_5d", "DHR_vol_20d", "PLD_Prologis_ret_5d", "GE_ret_1d", "EOG_EOGResources_vol_20d", "XLF_Fin_vol_20d", "EWA_Australia_zscore_60d", "EWM_Malaysia_zscore_60d", "CI_Cigna_vol_20d", "LLY_zscore_60d", "Nikkei_Japan_zscore_60d", "BTI_BritishAmerican_ret_5d", "EWA_Australia_ret_1d", "EWC_Canada_zscore_60d", "Core_CPI_zscore_60d"], "is_new": true}, {"model_id": "new_h7_CALM_LightGBM_N20_t2", "algo": "LightGBM", "regime": "CALM", "horizon": 7, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "XOM_ret_20d", "LOW_Lowes_ret_5d", "HangSeng_HK_ret_5d", "VVIX_ret_20d", "Brent_Oil_FRED_ret_5d", "PFE_ret_1d", "T_ret_1d", "BLK_BlackRock_zscore_60d", "TXN_vol_20d", "US1Y_Rate_ret_20d", "JNJ_ret_1d", "SCHW_Schwab_ret_5d", "MS_MorganStanley_ret_5d", "EWQ_France_ret_20d", "FedFunds_zscore_60d", "DAX_Germany_vol_20d", "Nikkei_Japan_zscore_60d", "ES_Evergy_ret_1d"], "is_new": true}, {"model_id": "new_h7_CALM_LightGBM_N20_t3", "algo": "LightGBM", "regime": "CALM", "horizon": 7, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "GD_GeneralDynamics_zscore_60d", "AORD_AUS_zscore_60d", "heston_var_ev_h7", "EOG_EOGResources_vol_20d", "EWH_HongKong_ret_5d", "PAYX_Paychex_zscore_60d", "EWM_Malaysia_zscore_60d", "US3Y_Rate_ret_5d", "EWL_Switzerland_vol_20d", "BA_ret_1d", "MO_AltriaMG_ret_1d", "PG_ret_20d", "HD_ret_1d", "CPB_CampbellSoup_ret_20d", "DAX_Germany_zscore_60d", "XLV_Health_zscore_60d", "US1Y_Rate_ret_5d", "Nikkei_Japan_vol_20d"], "is_new": true}, {"model_id": "new_h7_CALM_LightGBM_N20_t4", "algo": "LightGBM", "regime": "CALM", "horizon": 7, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "US3M_Rate_vol_20d", "AORD_AUS_zscore_60d", "IYR_US_REIT2_zscore_60d", "EXC_Exelon_zscore_60d", "EFFR_ret_1d", "INTC_ret_5d", "EWL_Switzerland_zscore_60d", "CI_Cigna_vol_20d", "QQQ_vol_20d", "heston_ev_h3", "AMZN_ret_5d", "DE_Deere_ret_5d", "SCHW_Schwab_ret_5d", "PG_ret_20d", "SBUX_vol_20d", "EOG_EOGResources_vol_20d", "CPB_CampbellSoup_ret_5d", "gjr_condvar_h1"], "is_new": true}, {"model_id": "new_h7_CALM_LightGBM_N20_t5", "algo": "LightGBM", "regime": "CALM", "horizon": 7, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "TM_Telephone_vol_20d", "PCAR_PaccarInc_ret_5d", "SLB_Schlumberger_ret_5d", "EFFR_ret_1d", "MS_MorganStanley_ret_1d", "JNJ_ret_1d", "AMD_ret_1d", "AXP_Amex_vol_20d", "XOM_ret_20d", "EWG_Germany_vol_20d", "CCI_CrownCastle_vol_20d", "EOG_EOGResources_ret_5d", "EWH_HongKong_ret_5d", "MRK_Merck_zscore_60d", "HD_ret_20d", "SO_SouthernCo_ret_5d", "EWM_Malaysia_zscore_60d", "ORCL_vol_20d"], "is_new": true}, {"model_id": "new_h7_CALM_LightGBM_N20_t6", "algo": "LightGBM", "regime": "CALM", "horizon": 7, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "XLY_Disc_vol_20d", "MSTR_Bitcoin3_ret_1d", "SBUX_zscore_60d", "CTAS_Cintas_vol_20d", "T_ret_1d", "EWH_HongKong_ret_5d", "DIS_vol_20d", "EWL_Switzerland_vol_20d", "VOD_Vodafone_zscore_60d", "IYM_BasicMaterials_ret_20d", "EWC_Canada_zscore_60d", "3M_vol_20d", "Brent_Oil_FRED_ret_20d", "JNJ_ret_1d", "Industrial_Production_zscore_60d", "US3M_Rate_zscore_60d", "LMT_LockheedMartin_ret_1d", "ES_Evergy_ret_1d"], "is_new": true}, {"model_id": "new_h7_CALM_LightGBM_N20_t7", "algo": "LightGBM", "regime": "CALM", "horizon": 7, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "Industrial_Production_zscore_60d", "VVIX_ret_20d", "spx_vol_5d", "CCI_CrownCastle_vol_20d", "ASX_Australia_ret_5d", "LOW_Lowes_ret_20d", "CMCSA_ret_1d", "TXN_vol_20d", "IYR_US_REIT2_zscore_60d", "AVB_AvalonBay_zscore_60d", "spx_abs_ret_max_5d", "SBUX_ret_5d", "US3M_Rate_zscore_60d", "DAX_Germany_zscore_60d", "EMR_Emerson_ret_20d", "SLB_Schlumberger_ret_5d", "CPB_CampbellSoup_ret_5d", "AXP_Amex_vol_20d"], "is_new": true}, {"model_id": "new_h7_CALM_LightGBM_N25_t0", "algo": "LightGBM", "regime": "CALM", "horizon": 7, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "SPY_zscore_60d", "WTI_Oil_FRED_zscore_60d", "US30Y_Rate_ret_20d", "IYR_US_REIT2_zscore_60d", "AXP_Amex_vol_20d", "AVB_AvalonBay_zscore_60d", "spx_abs_ret_max_5d", "EWL_Switzerland_zscore_60d", "TED_Spread_zscore_60d", "T_ret_1d", "gjr_condvar_h1", "XLK_Tech_zscore_60d", "CPB_CampbellSoup_ret_5d", "XOM_ret_1d", "heston_var_ev_h5", "DAX_Germany_zscore_60d", "ASX_Australia_ret_5d", "EWM_Malaysia_zscore_60d", "M_Macys_vol_20d", "AMT_AmericanTower_ret_1d", "US3M_Rate_vol_20d", "MS_MorganStanley_zscore_60d", "AORD_AUS_zscore_60d"], "is_new": true}, {"model_id": "new_h7_CALM_LightGBM_N25_t1", "algo": "LightGBM", "regime": "CALM", "horizon": 7, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "US3M_Rate_vol_20d", "M_Macys_vol_20d", "Brent_Oil_FRED_ret_20d", "EOG_EOGResources_ret_5d", "vix_acceleration_1d", "US6M_Rate_ret_20d", "3M_ret_5d", "PAYX_Paychex_zscore_60d", "DE_Deere_ret_5d", "SJM_JM_Smucker_ret_5d", "BLK_BlackRock_zscore_60d", "PPL_PPL_ret_1d", "XLB_Materials_zscore_60d", "EWY_Korea_ret_20d", "HD_ret_5d", "HD_zscore_60d", "Nikkei_Japan_vol_20d", "MSTR_Bitcoin3_ret_1d", "heston_var_ev_h7", "hmm_p_stress", "CTAS_Cintas_vol_20d", "AXP_Amex_vol_20d", "EFFR_vol_20d"], "is_new": true}, {"model_id": "new_h7_CALM_LightGBM_N25_t2", "algo": "LightGBM", "regime": "CALM", "horizon": 7, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "US6M_Rate_ret_20d", "heston_var_ev_h3", "SLB_Schlumberger_ret_1d", "BLK_BlackRock_zscore_60d", "GE_ret_1d", "GD_GeneralDynamics_zscore_60d", "Michigan_Sentiment_ret_20d", "3M_ret_5d", "3M_vol_20d", "SCHW_Schwab_ret_5d", "US5Y_Rate_ret_5d", "CCI_CrownCastle_vol_20d", "XLB_Materials_zscore_60d", "EWY_Korea_zscore_60d", "VOD_Vodafone_zscore_60d", "EOG_EOGResources_ret_5d", "EWL_Switzerland_vol_20d", "IYM_BasicMaterials_ret_20d", "EFFR_vol_20d", "spx_momentum_3d", "EQIX_Equinix_ret_5d", "SBUX_vol_20d", "EWH_HongKong_ret_5d"], "is_new": true}, {"model_id": "new_h7_CALM_LightGBM_N25_t3", "algo": "LightGBM", "regime": "CALM", "horizon": 7, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "TED_Spread_zscore_60d", "ASX_Australia_ret_5d", "IYM_BasicMaterials_ret_20d", "ITT_ITTInc_ret_5d", "AXP_Amex_ret_20d", "CPB_CampbellSoup_zscore_60d", "HangSeng_HK_ret_1d", "HD_ret_1d", "EWL_Switzerland_zscore_60d", "3M_ret_5d", "CPB_CampbellSoup_ret_5d", "CPB_CampbellSoup_vol_20d", "BTI_BritishAmerican_ret_5d", "Nikkei_Japan_vol_20d", "SBUX_vol_20d", "AORD_AUS_zscore_60d", "LLY_zscore_60d", "XOM_ret_20d", "DHR_ret_1d", "Brent_Oil_FRED_ret_20d", "BDX_Becton_Dickinson_ret_20d", "QQQ_vol_20d", "INTC_ret_1d"], "is_new": true}, {"model_id": "new_h7_CALM_LightGBM_N25_t4", "algo": "LightGBM", "regime": "CALM", "horizon": 7, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "HUM_Humana_ret_5d", "ENB_EnbridgeInc_ret_1d", "EWY_Korea_ret_20d", "Nikkei_Japan_vol_20d", "MO_AltriaMG_ret_1d", "EWH_HongKong_ret_5d", "Retail_Sales_zscore_60d", "LMT_LockheedMartin_ret_1d", "MRK_Merck_zscore_60d", "TGT_Target_zscore_60d", "FedFunds_zscore_60d", "XLF_Fin_vol_20d", "CPB_CampbellSoup_ret_20d", "DAX_Germany_zscore_60d", "T_ret_1d", "EQIX_Equinix_ret_5d", "MSTR_Bitcoin3_ret_5d", "IBEX_Spain_ret_20d", "EQR_Equity_ret_1d", "HangSeng_HK_vol_20d", "US3M_Rate_zscore_60d", "CCI_CrownCastle_vol_20d", "GD_GeneralDynamics_zscore_60d"], "is_new": true}, {"model_id": "new_h7_CALM_LightGBM_N25_t5", "algo": "LightGBM", "regime": "CALM", "horizon": 7, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "SLB_Schlumberger_ret_1d", "XLV_Health_zscore_60d", "HUM_Humana_ret_5d", "Michigan_Sentiment_ret_20d", "ITT_ITTInc_ret_5d", "TGT_Target_zscore_60d", "Industrial_Production_zscore_60d", "EXC_Exelon_ret_1d", "ORCL_zscore_60d", "EWM_Malaysia_ret_1d", "XLY_Disc_vol_20d", "US1Y_Rate_ret_20d", "US3M_Rate_zscore_60d", "EWQ_France_zscore_60d", "AMD_ret_5d", "CI_Cigna_vol_20d", "Nikkei_Japan_vol_20d", "Core_PCE_zscore_60d", "EQIX_Equinix_ret_5d", "PPL_PPL_ret_1d", "DHR_vol_20d", "HangSeng_HK_ret_1d", "EQR_Equity_ret_1d"], "is_new": true}, {"model_id": "new_h7_CALM_LightGBM_N25_t6", "algo": "LightGBM", "regime": "CALM", "horizon": 7, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "ITT_ITTInc_ret_5d", "AMT_AmericanTower_ret_1d", "PAYX_Paychex_vol_20d", "Core_PCE_zscore_60d", "NVDA_vol_20d", "TGT_Target_zscore_60d", "US3M_Rate_zscore_60d", "DAX_Germany_zscore_60d", "EWG_Germany_vol_20d", "DIS_vol_20d", "spx_momentum_3d", "SBUX_vol_20d", "CTAS_Cintas_vol_20d", "CI_Cigna_vol_20d", "HD_ret_1d", "MSTR_Bitcoin3_ret_1d", "PG_ret_20d", "INTC_ret_5d", "EWH_HongKong_ret_5d", "ASX_Australia_ret_5d", "CMCSA_ret_1d", "INTC_ret_1d", "XOM_ret_20d"], "is_new": true}, {"model_id": "new_h7_CALM_LightGBM_N25_t7", "algo": "LightGBM", "regime": "CALM", "horizon": 7, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "ASX_Australia_vol_20d", "CPB_CampbellSoup_vol_20d", "GILD_Gilead_ret_20d", "EWS_Singapore_ret_5d", "AMT_AmericanTower_ret_1d", "DIS_vol_20d", "PAYX_Paychex_ret_20d", "MS_MorganStanley_ret_5d", "PAYX_Paychex_zscore_60d", "XLY_Disc_vol_20d", "US3Y_Rate_ret_5d", "DAX_Germany_vol_20d", "SLB_Schlumberger_ret_5d", "DAX_Germany_zscore_60d", "spx_vol_5d", "XLV_Health_zscore_60d", "AMD_ret_1d", "HD_ret_1d", "DHR_vol_20d", "TGT_Target_zscore_60d", "EWM_Malaysia_ret_1d", "heston_ev_h3", "EWY_Korea_zscore_60d"], "is_new": true}, {"model_id": "new_h7_CALM_LightGBM_N30_t0", "algo": "LightGBM", "regime": "CALM", "horizon": 7, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "GD_GeneralDynamics_zscore_60d", "CCI_CrownCastle_vol_20d", "ASX_Australia_vol_20d", "SLB_Schlumberger_ret_5d", "HUM_Humana_ret_5d", "DAX_Germany_zscore_60d", "WTI_Oil_FRED_zscore_60d", "PLD_Prologis_ret_5d", "XLF_Fin_vol_20d", "NEE_NextEra_ret_20d", "QQQ_vol_20d", "MS_MorganStanley_ret_5d", "spx_abs_ret_max_5d", "DHR_ret_1d", "NFCI_ret_5d", "EWY_Korea_ret_20d", "EWM_Malaysia_vol_20d", "EWH_HongKong_ret_5d", "DAX_Germany_vol_20d", "EWG_Germany_ret_20d", "EFFR_vol_20d", "LOW_Lowes_ret_20d", "NOC_Northrop_ret_20d", "EQR_Equity_ret_1d", "CI_Cigna_vol_20d", "EOG_EOGResources_vol_20d", "heston_var_ev_h5", "GILD_Gilead_ret_20d"], "is_new": true}, {"model_id": "new_h7_CALM_LightGBM_N30_t1", "algo": "LightGBM", "regime": "CALM", "horizon": 7, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "MO_AltriaMG_ret_1d", "CPB_CampbellSoup_ret_5d", "ORCL_vol_20d", "DAX_Germany_zscore_60d", "XLB_Materials_zscore_60d", "PLD_Prologis_ret_5d", "hmm_p_stress", "DOW_Price_zscore_60d", "XLK_Tech_zscore_60d", "BTI_BritishAmerican_ret_5d", "Michigan_Sentiment_ret_20d", "DAX_Germany_vol_20d", "NVDA_vol_20d", "VOD_Vodafone_zscore_60d", "EWL_Switzerland_zscore_60d", "HD_ret_1d", "US5Y_Rate_ret_5d", "NOC_Northrop_ret_20d", "CLX_Clorox_vol_20d", "PFE_ret_1d", "IBEX_Spain_ret_20d", "MS_MorganStanley_zscore_60d", "EWY_Korea_zscore_60d", "T_ret_1d", "PAYX_Paychex_zscore_60d", "SJM_JM_Smucker_ret_5d", "ES_Evergy_ret_1d", "BDX_Becton_Dickinson_ret_20d"], "is_new": true}, {"model_id": "new_h7_CALM_LightGBM_N30_t2", "algo": "LightGBM", "regime": "CALM", "horizon": 7, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "SJM_JM_Smucker_ret_1d", "Industrial_Production_zscore_60d", "PPL_PPL_ret_1d", "AMGN_Amgen_ret_1d", "GD_GeneralDynamics_zscore_60d", "XOM_ret_20d", "MSTR_Bitcoin3_ret_5d", "GILD_Gilead_ret_20d", "TGT_Target_zscore_60d", "EWG_Germany_ret_20d", "PG_ret_20d", "CCI_CrownCastle_vol_20d", "INTC_ret_5d", "EWH_HongKong_ret_5d", "BLK_BlackRock_zscore_60d", "EWA_Australia_zscore_60d", "SBUX_vol_20d", "NOC_Northrop_ret_20d", "EWA_Australia_ret_1d", "CTAS_Cintas_vol_20d", "XLF_Fin_vol_20d", "HD_ret_1d", "SLB_Schlumberger_ret_1d", "CPB_CampbellSoup_vol_20d", "NEE_NextEra_ret_20d", "PFE_ret_1d", "EWY_Korea_zscore_60d", "EWM_Malaysia_ret_1d"], "is_new": true}, {"model_id": "new_h7_CALM_LightGBM_N30_t3", "algo": "LightGBM", "regime": "CALM", "horizon": 7, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "PFE_ret_1d", "DOW_Price_zscore_60d", "LMT_LockheedMartin_ret_1d", "US3M_Rate_vol_20d", "NVDA_vol_20d", "PCAR_PaccarInc_ret_5d", "EWM_Malaysia_ret_1d", "HUM_Humana_ret_5d", "SJM_JM_Smucker_ret_1d", "INTC_ret_5d", "EWA_Australia_ret_1d", "CPB_CampbellSoup_vol_20d", "Retail_Sales_zscore_60d", "US30Y_Rate_ret_20d", "heston_var_ev_h5", "CCI_CrownCastle_vol_20d", "NWL_Newell_ret_20d", "EWY_Korea_zscore_60d", "PG_ret_20d", "Industrial_Production_zscore_60d", "MSTR_Bitcoin3_ret_1d", "AMT_AmericanTower_ret_1d", "CPB_CampbellSoup_zscore_60d", "IYM_BasicMaterials_ret_20d", "PLD_Prologis_ret_5d", "MO_AltriaMG_ret_1d", "GILD_Gilead_ret_20d", "XOM_ret_20d"], "is_new": true}, {"model_id": "new_h7_CALM_LightGBM_N30_t4", "algo": "LightGBM", "regime": "CALM", "horizon": 7, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "spx_vol_5d", "EXC_Exelon_zscore_60d", "DOW_Price_zscore_60d", "US30Y_Rate_ret_20d", "EQR_Equity_ret_1d", "EWM_Malaysia_zscore_60d", "GILD_Gilead_ret_20d", "DHR_ret_1d", "T_ret_1d", "EOG_EOGResources_ret_5d", "BLK_BlackRock_zscore_60d", "XLY_Disc_vol_20d", "ORCL_zscore_60d", "TED_Spread_vol_20d", "SJM_JM_Smucker_ret_5d", "NWL_Newell_ret_20d", "DAX_Germany_zscore_60d", "EFFR_ret_1d", "EWY_Korea_ret_20d", "IWM_SmallCap_vol_20d", "EWM_Malaysia_vol_20d", "heston_var_ev_h7", "EQIX_Equinix_ret_5d", "vix_mean_abs_ret_5d", "Core_PCE_zscore_60d", "EWM_Malaysia_ret_1d", "XLB_Materials_zscore_60d", "US1Y_Rate_ret_5d"], "is_new": true}, {"model_id": "new_h7_CALM_LightGBM_N30_t5", "algo": "LightGBM", "regime": "CALM", "horizon": 7, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "ENB_EnbridgeInc_ret_1d", "IWM_SmallCap_vol_20d", "US3M_Rate_zscore_60d", "US30Y_Rate_ret_20d", "heston_ev_h3", "VOD_Vodafone_zscore_60d", "MS_MorganStanley_ret_1d", "BA_ret_1d", "XLK_Tech_zscore_60d", "US5Y_Rate_ret_5d", "EWS_Singapore_ret_5d", "WTI_Oil_FRED_zscore_60d", "HD_ret_20d", "SCHW_Schwab_ret_5d", "LOW_Lowes_ret_20d", "TXN_vol_20d", "MRK_Merck_zscore_60d", "SLB_Schlumberger_ret_1d", "gjr_condvar_h1", "HangSeng_HK_ret_5d", "EXC_Exelon_zscore_60d", "EWQ_France_zscore_60d", "DOW_Price_zscore_60d", "SO_SouthernCo_ret_5d", "AXP_Amex_ret_20d", "CPB_CampbellSoup_ret_5d", "EXC_Exelon_ret_1d", "AMD_ret_5d"], "is_new": true}, {"model_id": "new_h7_CALM_LightGBM_N30_t6", "algo": "LightGBM", "regime": "CALM", "horizon": 7, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "ORCL_zscore_60d", "PCAR_PaccarInc_ret_5d", "VRP_ma5", "AMT_AmericanTower_ret_1d", "CPB_CampbellSoup_zscore_60d", "SBUX_zscore_60d", "Brent_Oil_FRED_ret_5d", "LLY_zscore_60d", "HD_ret_20d", "SJM_JM_Smucker_ret_5d", "US7Y_Rate_ret_20d", "FedFunds_zscore_60d", "US3M_Rate_vol_20d", "INTC_ret_5d", "BA_ret_1d", "MRK_Merck_zscore_60d", "EWY_Korea_ret_20d", "MSTR_Bitcoin3_ret_5d", "DAX_Germany_zscore_60d", "EWQ_France_zscore_60d", "Nikkei_Japan_zscore_60d", "AMD_ret_5d", "PAYX_Paychex_ret_20d", "NOC_Northrop_ret_20d", "US1Y_Rate_ret_5d", "EMR_Emerson_ret_20d", "spx_vol_5d", "EWM_Malaysia_vol_20d"], "is_new": true}, {"model_id": "new_h7_CALM_LightGBM_N30_t7", "algo": "LightGBM", "regime": "CALM", "horizon": 7, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "SJM_JM_Smucker_ret_5d", "IWM_SmallCap_vol_20d", "US3M_Rate_zscore_60d", "ORCL_vol_20d", "EWY_Korea_zscore_60d", "NFCI_ret_5d", "LUV_SouthwestAir_ret_5d", "DHR_ret_1d", "BTI_BritishAmerican_ret_5d", "IYM_BasicMaterials_ret_20d", "SBUX_ret_5d", "DIS_vol_20d", "HD_ret_5d", "TM_Telephone_ret_1d", "XLK_Tech_zscore_60d", "heston_ev_h3", "MSTR_Bitcoin3_ret_20d", "MS_MorganStanley_ret_5d", "vix_acceleration_1d", "HD_ret_1d", "NWL_Newell_ret_20d", "HD_ret_20d", "EWJ_Japan_vol_20d", "SLB_Schlumberger_ret_5d", "US3M_Rate_vol_20d", "PPL_PPL_ret_1d", "EWL_Switzerland_vol_20d", "3M_vol_20d"], "is_new": true}, {"model_id": "new_h7_CALM_GradientBoosting_N5_t0", "algo": "GradientBoosting", "regime": "CALM", "horizon": 7, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "HD_ret_1d", "MS_MorganStanley_zscore_60d", "NEE_NextEra_ret_20d"], "is_new": true}, {"model_id": "new_h7_CALM_GradientBoosting_N5_t1", "algo": "GradientBoosting", "regime": "CALM", "horizon": 7, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "DE_Deere_vol_20d", "NFCI_ret_5d", "CPB_CampbellSoup_vol_20d"], "is_new": true}, {"model_id": "new_h7_CALM_GradientBoosting_N5_t2", "algo": "GradientBoosting", "regime": "CALM", "horizon": 7, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWC_Canada_zscore_60d", "AMT_AmericanTower_ret_1d", "IYM_BasicMaterials_ret_20d"], "is_new": true}, {"model_id": "new_h7_CALM_GradientBoosting_N5_t3", "algo": "GradientBoosting", "regime": "CALM", "horizon": 7, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "AMZN_ret_5d", "US3M_Rate_vol_20d", "BTI_BritishAmerican_ret_20d"], "is_new": true}, {"model_id": "new_h7_CALM_GradientBoosting_N5_t4", "algo": "GradientBoosting", "regime": "CALM", "horizon": 7, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "MS_MorganStanley_zscore_60d", "Nikkei_Japan_zscore_60d", "US3M_Rate_vol_20d"], "is_new": true}, {"model_id": "new_h7_CALM_GradientBoosting_N5_t5", "algo": "GradientBoosting", "regime": "CALM", "horizon": 7, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "spx_abs_ret_max_5d", "AMT_AmericanTower_ret_1d", "EMR_Emerson_ret_20d"], "is_new": true}, {"model_id": "new_h7_CALM_GradientBoosting_N5_t6", "algo": "GradientBoosting", "regime": "CALM", "horizon": 7, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "AORD_AUS_zscore_60d", "VOD_Vodafone_zscore_60d", "CLX_Clorox_vol_20d"], "is_new": true}, {"model_id": "new_h7_CALM_GradientBoosting_N5_t7", "algo": "GradientBoosting", "regime": "CALM", "horizon": 7, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "SBUX_zscore_60d", "EWA_Australia_zscore_60d", "T10Y2Y_Spread_ret_5d"], "is_new": true}, {"model_id": "new_h7_CALM_GradientBoosting_N8_t0", "algo": "GradientBoosting", "regime": "CALM", "horizon": 7, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "HD_zscore_60d", "XLV_Health_zscore_60d", "SPY_zscore_60d", "EWA_Australia_zscore_60d", "INTC_ret_5d", "NEE_NextEra_ret_20d"], "is_new": true}, {"model_id": "new_h7_CALM_GradientBoosting_N8_t1", "algo": "GradientBoosting", "regime": "CALM", "horizon": 7, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "AXP_Amex_vol_20d", "EWJ_Japan_vol_20d", "Michigan_Sentiment_ret_20d", "BTI_BritishAmerican_ret_5d", "US7Y_Rate_ret_20d", "EWL_Switzerland_zscore_60d"], "is_new": true}, {"model_id": "new_h7_CALM_GradientBoosting_N8_t2", "algo": "GradientBoosting", "regime": "CALM", "horizon": 7, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "INTC_ret_5d", "PLD_Prologis_ret_5d", "Nikkei_Japan_vol_20d", "TXN_vol_20d", "PPL_PPL_ret_1d", "LMT_LockheedMartin_ret_1d"], "is_new": true}, {"model_id": "new_h7_CALM_GradientBoosting_N8_t3", "algo": "GradientBoosting", "regime": "CALM", "horizon": 7, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWY_Korea_ret_20d", "ORCL_zscore_60d", "VVIX_ret_20d", "SJM_JM_Smucker_ret_1d", "VRP_ma5", "SBUX_zscore_60d"], "is_new": true}, {"model_id": "new_h7_CALM_GradientBoosting_N8_t4", "algo": "GradientBoosting", "regime": "CALM", "horizon": 7, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "HangSeng_HK_ret_5d", "DE_Deere_ret_5d", "CPB_CampbellSoup_ret_5d", "PFE_ret_1d", "CMCSA_ret_1d", "XLB_Materials_zscore_60d"], "is_new": true}, {"model_id": "new_h7_CALM_GradientBoosting_N8_t5", "algo": "GradientBoosting", "regime": "CALM", "horizon": 7, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "US3M_Rate_zscore_60d", "US1Y_Rate_ret_5d", "EOG_EOGResources_ret_5d", "MSTR_Bitcoin3_ret_1d", "EWL_Switzerland_vol_20d", "IBEX_Spain_ret_20d"], "is_new": true}, {"model_id": "new_h7_CALM_GradientBoosting_N8_t6", "algo": "GradientBoosting", "regime": "CALM", "horizon": 7, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "IWM_SmallCap_vol_20d", "EWJ_Japan_vol_20d", "PG_ret_20d", "LUV_SouthwestAir_ret_5d", "heston_var_ev_h5", "heston_var_ev_h7"], "is_new": true}, {"model_id": "new_h7_CALM_GradientBoosting_N8_t7", "algo": "GradientBoosting", "regime": "CALM", "horizon": 7, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "Core_PCE_zscore_60d", "BTI_BritishAmerican_ret_20d", "EXC_Exelon_ret_1d", "CTAS_Cintas_vol_20d", "AMD_ret_1d", "CCI_CrownCastle_vol_20d"], "is_new": true}, {"model_id": "new_h7_CALM_GradientBoosting_N10_t0", "algo": "GradientBoosting", "regime": "CALM", "horizon": 7, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "US3Y_Rate_ret_5d", "AMGN_Amgen_ret_1d", "LMT_LockheedMartin_ret_1d", "XLY_Disc_vol_20d", "HangSeng_HK_vol_20d", "PAYX_Paychex_zscore_60d", "CPB_CampbellSoup_ret_20d", "EWG_Germany_ret_20d"], "is_new": true}, {"model_id": "new_h7_CALM_GradientBoosting_N10_t1", "algo": "GradientBoosting", "regime": "CALM", "horizon": 7, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "MSTR_Bitcoin3_ret_20d", "NWL_Newell_ret_20d", "VOD_Vodafone_zscore_60d", "MRK_Merck_zscore_60d", "XLK_Tech_zscore_60d", "SLB_Schlumberger_ret_5d", "Brent_Oil_FRED_ret_20d", "hmm_p_stress"], "is_new": true}, {"model_id": "new_h7_CALM_GradientBoosting_N10_t2", "algo": "GradientBoosting", "regime": "CALM", "horizon": 7, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWA_Australia_ret_1d", "US6M_Rate_ret_20d", "GD_GeneralDynamics_zscore_60d", "SCHW_Schwab_ret_5d", "EWL_Switzerland_zscore_60d", "CPB_CampbellSoup_zscore_60d", "ES_Evergy_ret_1d", "XOM_ret_20d"], "is_new": true}, {"model_id": "new_h7_CALM_GradientBoosting_N10_t3", "algo": "GradientBoosting", "regime": "CALM", "horizon": 7, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "SCHW_Schwab_ret_5d", "LMT_LockheedMartin_ret_1d", "CPB_CampbellSoup_ret_20d", "HUM_Humana_ret_5d", "US3M_Rate_vol_20d", "XOM_ret_20d", "NVDA_vol_20d", "IBEX_Spain_ret_20d"], "is_new": true}, {"model_id": "new_h7_CALM_GradientBoosting_N10_t4", "algo": "GradientBoosting", "regime": "CALM", "horizon": 7, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "SBUX_vol_20d", "EWY_Korea_zscore_60d", "CPB_CampbellSoup_ret_20d", "NOC_Northrop_ret_20d", "AORD_AUS_zscore_60d", "PCAR_PaccarInc_ret_5d", "EWQ_France_zscore_60d", "CMCSA_ret_1d"], "is_new": true}, {"model_id": "new_h7_CALM_GradientBoosting_N10_t5", "algo": "GradientBoosting", "regime": "CALM", "horizon": 7, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "QQQ_vol_20d", "TGT_Target_zscore_60d", "US30Y_Rate_ret_20d", "DHR_ret_1d", "Industrial_Production_zscore_60d", "LOW_Lowes_ret_5d", "MSTR_Bitcoin3_ret_5d", "NVDA_vol_20d"], "is_new": true}, {"model_id": "new_h7_CALM_GradientBoosting_N10_t6", "algo": "GradientBoosting", "regime": "CALM", "horizon": 7, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "AXP_Amex_vol_20d", "T10Y2Y_Spread_ret_5d", "DOW_Price_zscore_60d", "US1Y_Rate_ret_5d", "vix_mean_abs_ret_5d", "PAYX_Paychex_zscore_60d", "DE_Deere_ret_5d", "PAYX_Paychex_ret_20d"], "is_new": true}, {"model_id": "new_h7_CALM_GradientBoosting_N10_t7", "algo": "GradientBoosting", "regime": "CALM", "horizon": 7, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "IBEX_Spain_ret_20d", "SJM_JM_Smucker_ret_1d", "AXP_Amex_ret_20d", "TGT_Target_zscore_60d", "TED_Spread_vol_20d", "PAYX_Paychex_vol_20d", "ASX_Australia_ret_5d", "EXC_Exelon_ret_1d"], "is_new": true}, {"model_id": "new_h7_CALM_GradientBoosting_N12_t0", "algo": "GradientBoosting", "regime": "CALM", "horizon": 7, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "SBUX_ret_5d", "EWA_Australia_ret_1d", "AMGN_Amgen_ret_1d", "DOW_Price_zscore_60d", "Retail_Sales_zscore_60d", "LOW_Lowes_ret_5d", "heston_var_ev_h5", "TGT_Target_zscore_60d", "SJM_JM_Smucker_ret_1d", "VOD_Vodafone_zscore_60d"], "is_new": true}, {"model_id": "new_h7_CALM_GradientBoosting_N12_t1", "algo": "GradientBoosting", "regime": "CALM", "horizon": 7, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "LOW_Lowes_ret_20d", "AXP_Amex_ret_20d", "AORD_AUS_zscore_60d", "ASX_Australia_vol_20d", "VVIX_ret_20d", "vix_acceleration_1d", "GD_GeneralDynamics_zscore_60d", "vix_mean_abs_ret_5d", "EWL_Switzerland_vol_20d", "MS_MorganStanley_ret_1d"], "is_new": true}, {"model_id": "new_h7_CALM_GradientBoosting_N12_t2", "algo": "GradientBoosting", "regime": "CALM", "horizon": 7, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "Industrial_Production_zscore_60d", "HD_ret_1d", "LLY_zscore_60d", "FedFunds_zscore_60d", "ES_Evergy_ret_1d", "EOG_EOGResources_vol_20d", "EWJ_Japan_vol_20d", "ORCL_vol_20d", "AMD_ret_1d", "NOC_Northrop_ret_20d"], "is_new": true}, {"model_id": "new_h7_CALM_GradientBoosting_N12_t3", "algo": "GradientBoosting", "regime": "CALM", "horizon": 7, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "PAYX_Paychex_ret_20d", "DE_Deere_vol_20d", "US7Y_Rate_ret_20d", "Nikkei_Japan_vol_20d", "US6M_Rate_ret_20d", "spx_vol_5d", "EWL_Switzerland_zscore_60d", "EFFR_vol_20d", "CPB_CampbellSoup_vol_20d", "IYM_BasicMaterials_ret_20d"], "is_new": true}, {"model_id": "new_h7_CALM_GradientBoosting_N12_t4", "algo": "GradientBoosting", "regime": "CALM", "horizon": 7, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "PAYX_Paychex_zscore_60d", "XOM_ret_20d", "ASX_Australia_vol_20d", "SLB_Schlumberger_ret_5d", "TED_Spread_vol_20d", "LOW_Lowes_ret_20d", "GD_GeneralDynamics_zscore_60d", "TXN_vol_20d", "CTAS_Cintas_vol_20d", "US30Y_Rate_ret_20d"], "is_new": true}, {"model_id": "new_h7_CALM_GradientBoosting_N12_t5", "algo": "GradientBoosting", "regime": "CALM", "horizon": 7, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "XLB_Materials_zscore_60d", "spx_vol_5d", "gjr_condvar_h1", "AMD_ret_5d", "VOD_Vodafone_zscore_60d", "EWY_Korea_zscore_60d", "CI_Cigna_vol_20d", "EQR_Equity_ret_1d", "MO_AltriaMG_ret_1d", "HangSeng_HK_vol_20d"], "is_new": true}, {"model_id": "new_h7_CALM_GradientBoosting_N12_t6", "algo": "GradientBoosting", "regime": "CALM", "horizon": 7, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWG_Germany_vol_20d", "EWJ_Japan_vol_20d", "ASX_Australia_ret_5d", "EWC_Canada_zscore_60d", "HangSeng_HK_ret_5d", "Industrial_Production_zscore_60d", "TM_Telephone_ret_1d", "HD_ret_5d", "3M_vol_20d", "VOD_Vodafone_zscore_60d"], "is_new": true}, {"model_id": "new_h7_CALM_GradientBoosting_N12_t7", "algo": "GradientBoosting", "regime": "CALM", "horizon": 7, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "SLB_Schlumberger_ret_1d", "US3Y_Rate_ret_5d", "HUM_Humana_ret_5d", "EWY_Korea_ret_20d", "heston_var_ev_h3", "ORCL_vol_20d", "NFCI_ret_5d", "NVDA_vol_20d", "SCHW_Schwab_ret_5d", "ASX_Australia_vol_20d"], "is_new": true}, {"model_id": "new_h7_CALM_GradientBoosting_N15_t0", "algo": "GradientBoosting", "regime": "CALM", "horizon": 7, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "US6M_Rate_ret_20d", "US3Y_Rate_ret_5d", "M_Macys_vol_20d", "Core_PCE_zscore_60d", "EWM_Malaysia_ret_1d", "AXP_Amex_ret_20d", "ORCL_vol_20d", "VVIX_ret_20d", "LOW_Lowes_ret_5d", "SBUX_ret_5d", "BA_ret_1d", "PPL_PPL_ret_1d", "DAX_Germany_vol_20d"], "is_new": true}, {"model_id": "new_h7_CALM_GradientBoosting_N15_t1", "algo": "GradientBoosting", "regime": "CALM", "horizon": 7, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "spx_momentum_3d", "spx_abs_ret_max_5d", "SLB_Schlumberger_ret_5d", "Retail_Sales_zscore_60d", "TED_Spread_zscore_60d", "INTC_ret_5d", "EFFR_vol_20d", "SCHW_Schwab_ret_5d", "DE_Deere_vol_20d", "GD_GeneralDynamics_zscore_60d", "IBEX_Spain_ret_20d", "AORD_AUS_zscore_60d", "NOC_Northrop_ret_20d"], "is_new": true}, {"model_id": "new_h7_CALM_GradientBoosting_N15_t2", "algo": "GradientBoosting", "regime": "CALM", "horizon": 7, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "Brent_Oil_FRED_ret_20d", "SJM_JM_Smucker_ret_1d", "CCI_CrownCastle_vol_20d", "PLD_Prologis_ret_5d", "DAX_Germany_vol_20d", "Core_PCE_zscore_60d", "US1Y_Rate_ret_20d", "SLB_Schlumberger_ret_5d", "DAX_Germany_zscore_60d", "DHR_vol_20d", "Brent_Oil_FRED_ret_5d", "EWY_Korea_ret_20d", "PAYX_Paychex_ret_20d"], "is_new": true}, {"model_id": "new_h7_CALM_GradientBoosting_N15_t3", "algo": "GradientBoosting", "regime": "CALM", "horizon": 7, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWG_Germany_vol_20d", "DAX_Germany_zscore_60d", "heston_var_ev_h7", "TXN_vol_20d", "ASX_Australia_vol_20d", "AMT_AmericanTower_ret_1d", "AXP_Amex_ret_20d", "CPB_CampbellSoup_vol_20d", "WTI_Oil_FRED_zscore_60d", "IYR_US_REIT2_zscore_60d", "XLF_Fin_vol_20d", "HangSeng_HK_ret_1d", "3M_vol_20d"], "is_new": true}, {"model_id": "new_h7_CALM_GradientBoosting_N15_t4", "algo": "GradientBoosting", "regime": "CALM", "horizon": 7, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "AORD_AUS_zscore_60d", "EWL_Switzerland_vol_20d", "EWG_Germany_vol_20d", "EXC_Exelon_ret_1d", "NFCI_ret_5d", "PAYX_Paychex_ret_20d", "MO_AltriaMG_ret_1d", "PFE_ret_1d", "PAYX_Paychex_vol_20d", "EWG_Germany_ret_20d", "MS_MorganStanley_ret_5d", "vix_mean_abs_ret_5d", "GD_GeneralDynamics_zscore_60d"], "is_new": true}, {"model_id": "new_h7_CALM_GradientBoosting_N15_t5", "algo": "GradientBoosting", "regime": "CALM", "horizon": 7, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EQR_Equity_ret_1d", "MS_MorganStanley_ret_1d", "XLB_Materials_zscore_60d", "LMT_LockheedMartin_ret_1d", "NWL_Newell_ret_20d", "EOG_EOGResources_vol_20d", "3M_vol_20d", "XOM_ret_1d", "GE_ret_1d", "SLB_Schlumberger_ret_5d", "HD_ret_20d", "heston_var_ev_h5", "MS_MorganStanley_ret_5d"], "is_new": true}, {"model_id": "new_h7_CALM_GradientBoosting_N15_t6", "algo": "GradientBoosting", "regime": "CALM", "horizon": 7, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWL_Switzerland_vol_20d", "US3M_Rate_vol_20d", "SJM_JM_Smucker_ret_5d", "TM_Telephone_vol_20d", "HD_zscore_60d", "EWA_Australia_zscore_60d", "QQQ_vol_20d", "PPL_PPL_ret_1d", "DAX_Germany_zscore_60d", "CPB_CampbellSoup_vol_20d", "Brent_Oil_FRED_ret_20d", "SLB_Schlumberger_ret_5d", "EWG_Germany_vol_20d"], "is_new": true}, {"model_id": "new_h7_CALM_GradientBoosting_N15_t7", "algo": "GradientBoosting", "regime": "CALM", "horizon": 7, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "SBUX_ret_5d", "INTC_ret_1d", "spx_abs_ret_max_5d", "EWQ_France_zscore_60d", "ASX_Australia_ret_5d", "XOM_ret_1d", "heston_var_ev_h5", "Retail_Sales_zscore_60d", "ENB_EnbridgeInc_ret_1d", "AMGN_Amgen_ret_1d", "EXC_Exelon_zscore_60d", "SJM_JM_Smucker_ret_5d", "Michigan_Sentiment_ret_20d"], "is_new": true}, {"model_id": "new_h7_CALM_GradientBoosting_N20_t0", "algo": "GradientBoosting", "regime": "CALM", "horizon": 7, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "HD_ret_1d", "Brent_Oil_FRED_ret_20d", "DAX_Germany_zscore_60d", "VVIX_ret_20d", "DHR_ret_1d", "IYM_BasicMaterials_ret_20d", "AVB_AvalonBay_zscore_60d", "NOC_Northrop_ret_20d", "VOD_Vodafone_zscore_60d", "US1Y_Rate_ret_5d", "heston_var_ev_h7", "EWQ_France_ret_20d", "EQIX_Equinix_ret_5d", "BTI_BritishAmerican_ret_20d", "EXC_Exelon_ret_1d", "GILD_Gilead_ret_20d", "LUV_SouthwestAir_ret_5d", "spx_vol_5d"], "is_new": true}, {"model_id": "new_h7_CALM_GradientBoosting_N20_t1", "algo": "GradientBoosting", "regime": "CALM", "horizon": 7, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "CI_Cigna_vol_20d", "IBEX_Spain_ret_20d", "NEE_NextEra_ret_20d", "US1Y_Rate_ret_5d", "SBUX_vol_20d", "3M_vol_20d", "GILD_Gilead_ret_20d", "CCI_CrownCastle_vol_20d", "CPB_CampbellSoup_ret_20d", "LUV_SouthwestAir_ret_5d", "Core_PCE_zscore_60d", "NVDA_vol_20d", "US7Y_Rate_ret_20d", "Industrial_Production_zscore_60d", "INTC_ret_1d", "EWC_Canada_zscore_60d", "XLK_Tech_zscore_60d", "DE_Deere_vol_20d"], "is_new": true}, {"model_id": "new_h7_CALM_GradientBoosting_N20_t2", "algo": "GradientBoosting", "regime": "CALM", "horizon": 7, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "HD_ret_5d", "MS_MorganStanley_ret_5d", "EWY_Korea_zscore_60d", "spx_momentum_3d", "T10Y2Y_Spread_ret_5d", "US1Y_Rate_ret_5d", "AXP_Amex_vol_20d", "PAYX_Paychex_zscore_60d", "TM_Telephone_ret_1d", "WTI_Oil_FRED_zscore_60d", "XOM_ret_1d", "SJM_JM_Smucker_ret_1d", "BA_ret_1d", "PG_ret_20d", "SCHW_Schwab_ret_5d", "EWQ_France_ret_20d", "AMT_AmericanTower_ret_1d", "US5Y_Rate_ret_5d"], "is_new": true}, {"model_id": "new_h7_CALM_GradientBoosting_N20_t3", "algo": "GradientBoosting", "regime": "CALM", "horizon": 7, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EQR_Equity_ret_1d", "TED_Spread_vol_20d", "DIS_vol_20d", "Core_PCE_zscore_60d", "heston_ev_h3", "3M_vol_20d", "DHR_ret_1d", "LMT_LockheedMartin_ret_1d", "DOW_Price_zscore_60d", "EWC_Canada_zscore_60d", "GD_GeneralDynamics_zscore_60d", "HangSeng_HK_vol_20d", "DAX_Germany_vol_20d", "PFE_ret_1d", "CPB_CampbellSoup_zscore_60d", "US3M_Rate_vol_20d", "EWQ_France_ret_20d", "DAX_Germany_zscore_60d"], "is_new": true}, {"model_id": "new_h7_CALM_GradientBoosting_N20_t4", "algo": "GradientBoosting", "regime": "CALM", "horizon": 7, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "ORCL_zscore_60d", "PAYX_Paychex_zscore_60d", "WTI_Oil_FRED_zscore_60d", "CMCSA_ret_1d", "SO_SouthernCo_ret_5d", "LMT_LockheedMartin_vol_20d", "AVB_AvalonBay_zscore_60d", "M_Macys_vol_20d", "HangSeng_HK_vol_20d", "CPB_CampbellSoup_zscore_60d", "VVIX_ret_20d", "PCAR_PaccarInc_ret_5d", "SJM_JM_Smucker_ret_1d", "EWM_Malaysia_ret_1d", "DE_Deere_ret_5d", "AMZN_ret_5d", "EWL_Switzerland_vol_20d", "MS_MorganStanley_zscore_60d"], "is_new": true}, {"model_id": "new_h7_CALM_GradientBoosting_N20_t5", "algo": "GradientBoosting", "regime": "CALM", "horizon": 7, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "US6M_Rate_ret_20d", "T10Y2Y_Spread_ret_5d", "PCAR_PaccarInc_ret_5d", "ASX_Australia_vol_20d", "EWG_Germany_vol_20d", "PAYX_Paychex_zscore_60d", "TED_Spread_zscore_60d", "FedFunds_zscore_60d", "HD_ret_1d", "MRK_Merck_zscore_60d", "QQQ_vol_20d", "heston_ev_h3", "NEE_NextEra_ret_20d", "vix_acceleration_1d", "PLD_Prologis_ret_5d", "ORCL_vol_20d", "EWJ_Japan_vol_20d", "Core_PCE_zscore_60d"], "is_new": true}, {"model_id": "new_h7_CALM_GradientBoosting_N20_t6", "algo": "GradientBoosting", "regime": "CALM", "horizon": 7, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "CPB_CampbellSoup_ret_5d", "XLK_Tech_zscore_60d", "EOG_EOGResources_vol_20d", "XLY_Disc_vol_20d", "XLB_Materials_zscore_60d", "EWH_HongKong_ret_5d", "ES_Evergy_ret_1d", "EQIX_Equinix_ret_5d", "EWM_Malaysia_vol_20d", "US7Y_Rate_ret_20d", "HangSeng_HK_ret_1d", "SCHW_Schwab_ret_5d", "vix_acceleration_1d", "TGT_Target_zscore_60d", "ASX_Australia_vol_20d", "AVB_AvalonBay_zscore_60d", "AXP_Amex_vol_20d", "LMT_LockheedMartin_vol_20d"], "is_new": true}, {"model_id": "new_h7_CALM_GradientBoosting_N20_t7", "algo": "GradientBoosting", "regime": "CALM", "horizon": 7, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "LOW_Lowes_ret_5d", "MSTR_Bitcoin3_ret_1d", "spx_momentum_3d", "Michigan_Sentiment_ret_20d", "EWH_HongKong_ret_5d", "MSTR_Bitcoin3_ret_5d", "TGT_Target_zscore_60d", "IWM_SmallCap_vol_20d", "Nikkei_Japan_zscore_60d", "SPY_zscore_60d", "TED_Spread_vol_20d", "vix_acceleration_1d", "BLK_BlackRock_zscore_60d", "BTI_BritishAmerican_ret_20d", "ASX_Australia_vol_20d", "US6M_Rate_ret_20d", "gjr_condvar_h1", "XLF_Fin_vol_20d"], "is_new": true}, {"model_id": "new_h7_CALM_GradientBoosting_N25_t0", "algo": "GradientBoosting", "regime": "CALM", "horizon": 7, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "TGT_Target_zscore_60d", "ES_Evergy_ret_1d", "XLY_Disc_vol_20d", "AXP_Amex_vol_20d", "GILD_Gilead_ret_20d", "LOW_Lowes_ret_5d", "AMD_ret_1d", "hmm_p_stress", "SCHW_Schwab_ret_5d", "EWJ_Japan_vol_20d", "DAX_Germany_zscore_60d", "NEE_NextEra_ret_20d", "MS_MorganStanley_zscore_60d", "EWC_Canada_zscore_60d", "BLK_BlackRock_zscore_60d", "LOW_Lowes_ret_20d", "NOC_Northrop_ret_20d", "DOW_Price_zscore_60d", "XLK_Tech_zscore_60d", "VRP_ma5", "PAYX_Paychex_ret_20d", "DIS_vol_20d", "DE_Deere_ret_5d"], "is_new": true}, {"model_id": "new_h7_CALM_GradientBoosting_N25_t1", "algo": "GradientBoosting", "regime": "CALM", "horizon": 7, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "GD_GeneralDynamics_zscore_60d", "IBEX_Spain_ret_20d", "EFFR_vol_20d", "hmm_p_stress", "TM_Telephone_ret_1d", "SPY_zscore_60d", "EWY_Korea_zscore_60d", "Industrial_Production_zscore_60d", "TM_Telephone_vol_20d", "EWJ_Japan_vol_20d", "CMCSA_ret_1d", "Nikkei_Japan_vol_20d", "3M_vol_20d", "HD_ret_5d", "WTI_Oil_FRED_zscore_60d", "MS_MorganStanley_ret_1d", "AVB_AvalonBay_zscore_60d", "CPB_CampbellSoup_zscore_60d", "EWA_Australia_ret_1d", "EWG_Germany_vol_20d", "AMD_ret_1d", "SCHW_Schwab_ret_5d", "PG_ret_20d"], "is_new": true}, {"model_id": "new_h7_CALM_GradientBoosting_N25_t2", "algo": "GradientBoosting", "regime": "CALM", "horizon": 7, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "SLB_Schlumberger_ret_1d", "PCAR_PaccarInc_ret_5d", "LMT_LockheedMartin_ret_1d", "BA_ret_1d", "HUM_Humana_ret_5d", "LOW_Lowes_ret_20d", "EWM_Malaysia_zscore_60d", "DIS_vol_20d", "TED_Spread_zscore_60d", "DHR_ret_1d", "HangSeng_HK_ret_5d", "TM_Telephone_vol_20d", "MO_AltriaMG_ret_1d", "SO_SouthernCo_ret_5d", "BLK_BlackRock_zscore_60d", "heston_ev_h3", "IWM_SmallCap_vol_20d", "PFE_ret_1d", "Michigan_Sentiment_ret_20d", "US7Y_Rate_ret_20d", "US1Y_Rate_ret_20d", "MRK_Merck_zscore_60d", "XLY_Disc_vol_20d"], "is_new": true}, {"model_id": "new_h7_CALM_GradientBoosting_N25_t3", "algo": "GradientBoosting", "regime": "CALM", "horizon": 7, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "LLY_zscore_60d", "CPB_CampbellSoup_vol_20d", "AORD_AUS_zscore_60d", "TM_Telephone_ret_1d", "WTI_Oil_FRED_zscore_60d", "NVDA_vol_20d", "ORCL_vol_20d", "HangSeng_HK_ret_1d", "CLX_Clorox_vol_20d", "PG_ret_20d", "EOG_EOGResources_ret_5d", "TXN_vol_20d", "T_ret_1d", "ITT_ITTInc_ret_5d", "MO_AltriaMG_ret_1d", "CMCSA_ret_1d", "TGT_Target_zscore_60d", "US3M_Rate_vol_20d", "XLV_Health_zscore_60d", "IWM_SmallCap_vol_20d", "EWM_Malaysia_vol_20d", "HD_ret_1d", "vix_mean_abs_ret_5d"], "is_new": true}, {"model_id": "new_h7_CALM_GradientBoosting_N25_t4", "algo": "GradientBoosting", "regime": "CALM", "horizon": 7, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWH_HongKong_ret_5d", "TED_Spread_vol_20d", "LUV_SouthwestAir_ret_5d", "EOG_EOGResources_ret_5d", "CTAS_Cintas_vol_20d", "HUM_Humana_ret_5d", "NFCI_ret_5d", "DAX_Germany_vol_20d", "Brent_Oil_FRED_ret_5d", "CCI_CrownCastle_vol_20d", "BDX_Becton_Dickinson_ret_20d", "MO_AltriaMG_ret_1d", "LLY_zscore_60d", "VVIX_ret_20d", "PG_ret_20d", "US3M_Rate_zscore_60d", "HD_ret_1d", "ES_Evergy_ret_1d", "spx_momentum_3d", "EWY_Korea_ret_20d", "GILD_Gilead_ret_20d", "CPB_CampbellSoup_vol_20d", "EWS_Singapore_ret_5d"], "is_new": true}, {"model_id": "new_h7_CALM_GradientBoosting_N25_t5", "algo": "GradientBoosting", "regime": "CALM", "horizon": 7, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "ENB_EnbridgeInc_ret_1d", "SLB_Schlumberger_ret_5d", "XLV_Health_zscore_60d", "TM_Telephone_ret_1d", "XLB_Materials_zscore_60d", "EWJ_Japan_vol_20d", "MSTR_Bitcoin3_ret_1d", "INTC_ret_5d", "Core_CPI_zscore_60d", "spx_vol_5d", "LMT_LockheedMartin_ret_1d", "NFCI_ret_5d", "ITT_ITTInc_ret_5d", "GILD_Gilead_ret_20d", "LOW_Lowes_ret_20d", "NEE_NextEra_ret_20d", "3M_vol_20d", "EWM_Malaysia_ret_1d", "gjr_condvar_h1", "Michigan_Sentiment_ret_20d", "MSTR_Bitcoin3_ret_20d", "DAX_Germany_vol_20d", "SCHW_Schwab_ret_5d"], "is_new": true}, {"model_id": "new_h7_CALM_GradientBoosting_N25_t6", "algo": "GradientBoosting", "regime": "CALM", "horizon": 7, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "Retail_Sales_zscore_60d", "MS_MorganStanley_zscore_60d", "spx_momentum_3d", "XOM_ret_20d", "Industrial_Production_zscore_60d", "PLD_Prologis_ret_5d", "3M_vol_20d", "SBUX_ret_5d", "PAYX_Paychex_vol_20d", "VVIX_ret_20d", "FedFunds_zscore_60d", "US6M_Rate_ret_20d", "CPB_CampbellSoup_ret_20d", "EOG_EOGResources_vol_20d", "3M_ret_5d", "AXP_Amex_vol_20d", "EWA_Australia_ret_1d", "DE_Deere_ret_5d", "MO_AltriaMG_ret_1d", "SO_SouthernCo_ret_5d", "MSTR_Bitcoin3_ret_20d", "MSTR_Bitcoin3_ret_1d", "EWA_Australia_zscore_60d"], "is_new": true}, {"model_id": "new_h7_CALM_GradientBoosting_N25_t7", "algo": "GradientBoosting", "regime": "CALM", "horizon": 7, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "NFCI_ret_5d", "US3M_Rate_vol_20d", "SPY_zscore_60d", "QQQ_vol_20d", "AORD_AUS_zscore_60d", "vix_acceleration_1d", "BTI_BritishAmerican_ret_20d", "DOW_Price_zscore_60d", "ORCL_vol_20d", "NOC_Northrop_ret_20d", "PG_ret_20d", "CTAS_Cintas_vol_20d", "spx_momentum_3d", "EWL_Switzerland_vol_20d", "US1Y_Rate_ret_20d", "DE_Deere_vol_20d", "XLY_Disc_vol_20d", "EWG_Germany_vol_20d", "PPL_PPL_ret_1d", "SBUX_zscore_60d", "CPB_CampbellSoup_ret_20d", "EWJ_Japan_vol_20d", "AMZN_ret_5d"], "is_new": true}, {"model_id": "new_h7_CALM_GradientBoosting_N30_t0", "algo": "GradientBoosting", "regime": "CALM", "horizon": 7, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "CLX_Clorox_vol_20d", "EWL_Switzerland_vol_20d", "vix_acceleration_1d", "DIS_vol_20d", "Nikkei_Japan_zscore_60d", "US7Y_Rate_ret_20d", "AVB_AvalonBay_zscore_60d", "CMCSA_ret_1d", "TXN_vol_20d", "SBUX_zscore_60d", "XOM_ret_20d", "EWA_Australia_ret_1d", "NVDA_vol_20d", "EWS_Singapore_ret_5d", "SJM_JM_Smucker_ret_1d", "MSTR_Bitcoin3_ret_1d", "EQIX_Equinix_ret_5d", "SBUX_ret_5d", "HangSeng_HK_ret_1d", "EMR_Emerson_ret_20d", "AXP_Amex_vol_20d", "TM_Telephone_vol_20d", "M_Macys_vol_20d", "VRP_ma5", "ASX_Australia_vol_20d", "XLF_Fin_vol_20d", "CPB_CampbellSoup_vol_20d", "CTAS_Cintas_vol_20d"], "is_new": true}, {"model_id": "new_h7_CALM_GradientBoosting_N30_t1", "algo": "GradientBoosting", "regime": "CALM", "horizon": 7, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "heston_var_ev_h5", "HD_ret_20d", "GE_ret_1d", "heston_var_ev_h7", "PCAR_PaccarInc_ret_5d", "XLY_Disc_vol_20d", "NEE_NextEra_ret_20d", "T10Y2Y_Spread_ret_5d", "TM_Telephone_ret_1d", "EFFR_vol_20d", "EXC_Exelon_ret_1d", "TXN_vol_20d", "WTI_Oil_FRED_zscore_60d", "EWQ_France_zscore_60d", "DE_Deere_ret_5d", "EWC_Canada_zscore_60d", "CCI_CrownCastle_vol_20d", "PAYX_Paychex_vol_20d", "EXC_Exelon_zscore_60d", "TM_Telephone_vol_20d", "HangSeng_HK_ret_5d", "SLB_Schlumberger_ret_1d", "PPL_PPL_ret_1d", "FedFunds_zscore_60d", "CTAS_Cintas_vol_20d", "MRK_Merck_zscore_60d", "DHR_vol_20d", "AMD_ret_5d"], "is_new": true}, {"model_id": "new_h7_CALM_GradientBoosting_N30_t2", "algo": "GradientBoosting", "regime": "CALM", "horizon": 7, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "MRK_Merck_zscore_60d", "IBEX_Spain_ret_20d", "SPY_zscore_60d", "T10Y2Y_Spread_ret_5d", "AMZN_ret_5d", "MSTR_Bitcoin3_ret_20d", "CI_Cigna_vol_20d", "PPL_PPL_ret_1d", "HangSeng_HK_ret_5d", "SBUX_zscore_60d", "EWS_Singapore_ret_5d", "SO_SouthernCo_ret_5d", "EWQ_France_zscore_60d", "MS_MorganStanley_zscore_60d", "NWL_Newell_ret_20d", "HD_ret_20d", "SBUX_vol_20d", "CTAS_Cintas_vol_20d", "XOM_ret_20d", "EWH_HongKong_ret_5d", "XLK_Tech_zscore_60d", "HD_ret_1d", "TED_Spread_zscore_60d", "LMT_LockheedMartin_vol_20d", "hmm_p_stress", "HUM_Humana_ret_5d", "PG_ret_20d", "EFFR_ret_1d"], "is_new": true}, {"model_id": "new_h7_CALM_GradientBoosting_N30_t3", "algo": "GradientBoosting", "regime": "CALM", "horizon": 7, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "INTC_ret_1d", "MO_AltriaMG_ret_1d", "vix_mean_abs_ret_5d", "MS_MorganStanley_ret_5d", "3M_vol_20d", "JNJ_ret_1d", "VOD_Vodafone_zscore_60d", "EQIX_Equinix_ret_5d", "EWA_Australia_ret_1d", "EWG_Germany_ret_20d", "SPY_zscore_60d", "US6M_Rate_ret_20d", "DHR_ret_1d", "INTC_ret_5d", "CPB_CampbellSoup_zscore_60d", "NVDA_vol_20d", "US7Y_Rate_ret_20d", "DAX_Germany_vol_20d", "EWM_Malaysia_vol_20d", "AMZN_ret_5d", "NOC_Northrop_ret_20d", "TM_Telephone_vol_20d", "EWM_Malaysia_ret_1d", "CPB_CampbellSoup_vol_20d", "EQR_Equity_ret_1d", "XOM_ret_1d", "TED_Spread_vol_20d", "hmm_p_stress"], "is_new": true}, {"model_id": "new_h7_CALM_GradientBoosting_N30_t4", "algo": "GradientBoosting", "regime": "CALM", "horizon": 7, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWA_Australia_zscore_60d", "IYM_BasicMaterials_ret_20d", "XLV_Health_zscore_60d", "ASX_Australia_vol_20d", "SJM_JM_Smucker_ret_1d", "AXP_Amex_ret_20d", "MS_MorganStanley_ret_5d", "PLD_Prologis_ret_5d", "CLX_Clorox_vol_20d", "LOW_Lowes_ret_20d", "heston_var_ev_h3", "INTC_ret_5d", "EFFR_vol_20d", "LUV_SouthwestAir_ret_5d", "T_ret_1d", "MSTR_Bitcoin3_ret_5d", "EQR_Equity_ret_1d", "CTAS_Cintas_vol_20d", "SBUX_zscore_60d", "spx_momentum_3d", "AMD_ret_5d", "SLB_Schlumberger_ret_1d", "PPL_PPL_ret_1d", "VOD_Vodafone_zscore_60d", "BTI_BritishAmerican_ret_5d", "vix_mean_abs_ret_5d", "EOG_EOGResources_vol_20d", "CCI_CrownCastle_vol_20d"], "is_new": true}, {"model_id": "new_h7_CALM_GradientBoosting_N30_t5", "algo": "GradientBoosting", "regime": "CALM", "horizon": 7, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "heston_var_ev_h5", "PFE_ret_1d", "XLF_Fin_vol_20d", "TXN_vol_20d", "HangSeng_HK_ret_5d", "BLK_BlackRock_zscore_60d", "MO_AltriaMG_ret_1d", "SJM_JM_Smucker_ret_5d", "DHR_ret_1d", "SBUX_zscore_60d", "EWM_Malaysia_zscore_60d", "HD_ret_1d", "LUV_SouthwestAir_ret_5d", "XLB_Materials_zscore_60d", "XLK_Tech_zscore_60d", "T10Y2Y_Spread_ret_5d", "MSTR_Bitcoin3_ret_5d", "LMT_LockheedMartin_vol_20d", "T_ret_1d", "US7Y_Rate_ret_20d", "TGT_Target_zscore_60d", "VVIX_ret_20d", "EQIX_Equinix_ret_5d", "BDX_Becton_Dickinson_ret_20d", "WTI_Oil_FRED_zscore_60d", "LOW_Lowes_ret_20d", "TED_Spread_vol_20d", "EQR_Equity_ret_1d"], "is_new": true}, {"model_id": "new_h7_CALM_GradientBoosting_N30_t6", "algo": "GradientBoosting", "regime": "CALM", "horizon": 7, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWA_Australia_ret_1d", "XOM_ret_20d", "CPB_CampbellSoup_ret_20d", "Nikkei_Japan_zscore_60d", "EWQ_France_ret_20d", "AXP_Amex_ret_20d", "SBUX_ret_5d", "spx_abs_ret_max_5d", "EWH_HongKong_ret_5d", "LLY_zscore_60d", "VVIX_ret_20d", "VOD_Vodafone_zscore_60d", "TM_Telephone_vol_20d", "ITT_ITTInc_ret_5d", "PCAR_PaccarInc_ret_5d", "heston_ev_h3", "Brent_Oil_FRED_ret_20d", "DE_Deere_ret_5d", "EQIX_Equinix_ret_5d", "LMT_LockheedMartin_ret_1d", "EWG_Germany_ret_20d", "SCHW_Schwab_ret_5d", "NVDA_vol_20d", "SPY_zscore_60d", "IBEX_Spain_ret_20d", "T10Y2Y_Spread_ret_5d", "DHR_vol_20d", "T_ret_1d"], "is_new": true}, {"model_id": "new_h7_CALM_GradientBoosting_N30_t7", "algo": "GradientBoosting", "regime": "CALM", "horizon": 7, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "GE_ret_1d", "DE_Deere_ret_5d", "AMGN_Amgen_ret_1d", "spx_momentum_3d", "PAYX_Paychex_vol_20d", "CMCSA_ret_1d", "heston_var_ev_h7", "AVB_AvalonBay_zscore_60d", "CLX_Clorox_vol_20d", "XOM_ret_20d", "LLY_zscore_60d", "NVDA_vol_20d", "EWL_Switzerland_zscore_60d", "XOM_ret_1d", "US6M_Rate_ret_20d", "CPB_CampbellSoup_ret_5d", "T10Y2Y_Spread_ret_5d", "heston_var_ev_h5", "QQQ_vol_20d", "LMT_LockheedMartin_ret_1d", "SJM_JM_Smucker_ret_1d", "MSTR_Bitcoin3_ret_20d", "HD_ret_1d", "HD_ret_5d", "EWG_Germany_vol_20d", "EOG_EOGResources_vol_20d", "DIS_vol_20d", "EWC_Canada_zscore_60d"], "is_new": true}, {"model_id": "new_h7_CALM_RandomForest_N5_t0", "algo": "RandomForest", "regime": "CALM", "horizon": 7, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EMR_Emerson_ret_20d", "EWA_Australia_ret_1d", "DOW_Price_zscore_60d"], "is_new": true}, {"model_id": "new_h7_CALM_RandomForest_N5_t1", "algo": "RandomForest", "regime": "CALM", "horizon": 7, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "IBEX_Spain_ret_20d", "US3Y_Rate_ret_5d", "US30Y_Rate_ret_20d"], "is_new": true}, {"model_id": "new_h7_CALM_RandomForest_N5_t2", "algo": "RandomForest", "regime": "CALM", "horizon": 7, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "GE_ret_1d", "ASX_Australia_vol_20d", "NEE_NextEra_ret_20d"], "is_new": true}, {"model_id": "new_h7_CALM_RandomForest_N5_t3", "algo": "RandomForest", "regime": "CALM", "horizon": 7, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "spx_vol_5d", "EWJ_Japan_vol_20d", "Nikkei_Japan_zscore_60d"], "is_new": true}, {"model_id": "new_h7_CALM_RandomForest_N5_t4", "algo": "RandomForest", "regime": "CALM", "horizon": 7, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "US30Y_Rate_ret_20d", "EMR_Emerson_ret_20d", "EQR_Equity_ret_1d"], "is_new": true}, {"model_id": "new_h7_CALM_RandomForest_N5_t5", "algo": "RandomForest", "regime": "CALM", "horizon": 7, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "LUV_SouthwestAir_ret_5d", "MSTR_Bitcoin3_ret_1d", "WTI_Oil_FRED_zscore_60d"], "is_new": true}, {"model_id": "new_h7_CALM_RandomForest_N5_t6", "algo": "RandomForest", "regime": "CALM", "horizon": 7, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "PCAR_PaccarInc_ret_5d", "INTC_ret_5d", "MO_AltriaMG_ret_1d"], "is_new": true}, {"model_id": "new_h7_CALM_RandomForest_N5_t7", "algo": "RandomForest", "regime": "CALM", "horizon": 7, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWL_Switzerland_vol_20d", "ASX_Australia_ret_5d", "SO_SouthernCo_ret_5d"], "is_new": true}, {"model_id": "new_h7_CALM_RandomForest_N8_t0", "algo": "RandomForest", "regime": "CALM", "horizon": 7, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "hmm_p_stress", "EWM_Malaysia_vol_20d", "AORD_AUS_zscore_60d", "ASX_Australia_vol_20d", "CCI_CrownCastle_vol_20d", "AVB_AvalonBay_zscore_60d"], "is_new": true}, {"model_id": "new_h7_CALM_RandomForest_N8_t1", "algo": "RandomForest", "regime": "CALM", "horizon": 7, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWY_Korea_zscore_60d", "EXC_Exelon_ret_1d", "EWL_Switzerland_zscore_60d", "HangSeng_HK_ret_1d", "heston_var_ev_h3", "DHR_ret_1d"], "is_new": true}, {"model_id": "new_h7_CALM_RandomForest_N8_t2", "algo": "RandomForest", "regime": "CALM", "horizon": 7, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "MS_MorganStanley_ret_1d", "DAX_Germany_vol_20d", "EWA_Australia_zscore_60d", "LOW_Lowes_ret_5d", "HangSeng_HK_vol_20d", "MSTR_Bitcoin3_ret_5d"], "is_new": true}, {"model_id": "new_h7_CALM_RandomForest_N8_t3", "algo": "RandomForest", "regime": "CALM", "horizon": 7, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWA_Australia_zscore_60d", "SLB_Schlumberger_ret_5d", "PFE_ret_1d", "Industrial_Production_zscore_60d", "ORCL_vol_20d", "INTC_ret_1d"], "is_new": true}, {"model_id": "new_h7_CALM_RandomForest_N8_t4", "algo": "RandomForest", "regime": "CALM", "horizon": 7, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "CMCSA_ret_1d", "EOG_EOGResources_vol_20d", "XLF_Fin_vol_20d", "HUM_Humana_ret_5d", "heston_var_ev_h7", "gjr_condvar_h1"], "is_new": true}, {"model_id": "new_h7_CALM_RandomForest_N8_t5", "algo": "RandomForest", "regime": "CALM", "horizon": 7, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "Brent_Oil_FRED_ret_20d", "DHR_vol_20d", "Retail_Sales_zscore_60d", "DHR_ret_1d", "HangSeng_HK_vol_20d", "CTAS_Cintas_vol_20d"], "is_new": true}, {"model_id": "new_h7_CALM_RandomForest_N8_t6", "algo": "RandomForest", "regime": "CALM", "horizon": 7, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "DOW_Price_zscore_60d", "SLB_Schlumberger_ret_5d", "US1Y_Rate_ret_5d", "PFE_ret_1d", "EWC_Canada_zscore_60d", "EWS_Singapore_ret_5d"], "is_new": true}, {"model_id": "new_h7_CALM_RandomForest_N8_t7", "algo": "RandomForest", "regime": "CALM", "horizon": 7, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "LMT_LockheedMartin_ret_1d", "HangSeng_HK_ret_5d", "XLV_Health_zscore_60d", "TM_Telephone_ret_1d", "LMT_LockheedMartin_vol_20d", "NFCI_ret_5d"], "is_new": true}, {"model_id": "new_h7_CALM_RandomForest_N10_t0", "algo": "RandomForest", "regime": "CALM", "horizon": 7, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "QQQ_vol_20d", "PAYX_Paychex_zscore_60d", "vix_acceleration_1d", "PG_ret_20d", "US1Y_Rate_ret_20d", "EWC_Canada_zscore_60d", "EOG_EOGResources_ret_5d", "Brent_Oil_FRED_ret_20d"], "is_new": true}, {"model_id": "new_h7_CALM_RandomForest_N10_t1", "algo": "RandomForest", "regime": "CALM", "horizon": 7, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWG_Germany_vol_20d", "PAYX_Paychex_ret_20d", "DIS_vol_20d", "MSTR_Bitcoin3_ret_20d", "INTC_ret_1d", "Nikkei_Japan_zscore_60d", "TED_Spread_vol_20d", "EWG_Germany_ret_20d"], "is_new": true}, {"model_id": "new_h7_CALM_RandomForest_N10_t2", "algo": "RandomForest", "regime": "CALM", "horizon": 7, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "HD_ret_5d", "INTC_ret_1d", "BTI_BritishAmerican_ret_20d", "MS_MorganStanley_ret_5d", "MO_AltriaMG_ret_1d", "LOW_Lowes_ret_20d", "XLK_Tech_zscore_60d", "IWM_SmallCap_vol_20d"], "is_new": true}, {"model_id": "new_h7_CALM_RandomForest_N10_t3", "algo": "RandomForest", "regime": "CALM", "horizon": 7, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWM_Malaysia_zscore_60d", "EWL_Switzerland_zscore_60d", "Industrial_Production_zscore_60d", "BA_ret_1d", "ES_Evergy_ret_1d", "Core_CPI_zscore_60d", "HangSeng_HK_vol_20d", "EMR_Emerson_ret_20d"], "is_new": true}, {"model_id": "new_h7_CALM_RandomForest_N10_t4", "algo": "RandomForest", "regime": "CALM", "horizon": 7, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "XLV_Health_zscore_60d", "VVIX_ret_20d", "T10Y2Y_Spread_ret_5d", "WTI_Oil_FRED_zscore_60d", "DE_Deere_ret_5d", "EWL_Switzerland_zscore_60d", "PAYX_Paychex_ret_20d", "DHR_ret_1d"], "is_new": true}, {"model_id": "new_h7_CALM_RandomForest_N10_t5", "algo": "RandomForest", "regime": "CALM", "horizon": 7, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWC_Canada_zscore_60d", "HUM_Humana_ret_5d", "EXC_Exelon_ret_1d", "DHR_vol_20d", "XOM_ret_1d", "BDX_Becton_Dickinson_ret_20d", "ORCL_zscore_60d", "Core_PCE_zscore_60d"], "is_new": true}, {"model_id": "new_h7_CALM_RandomForest_N10_t6", "algo": "RandomForest", "regime": "CALM", "horizon": 7, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EFFR_vol_20d", "Industrial_Production_zscore_60d", "EWJ_Japan_vol_20d", "vix_mean_abs_ret_5d", "AMD_ret_5d", "3M_ret_5d", "PPL_PPL_ret_1d", "EFFR_ret_1d"], "is_new": true}, {"model_id": "new_h7_CALM_RandomForest_N10_t7", "algo": "RandomForest", "regime": "CALM", "horizon": 7, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "GD_GeneralDynamics_zscore_60d", "EXC_Exelon_ret_1d", "HD_ret_1d", "BLK_BlackRock_zscore_60d", "Industrial_Production_zscore_60d", "CPB_CampbellSoup_ret_5d", "EWC_Canada_zscore_60d", "spx_vol_5d"], "is_new": true}, {"model_id": "new_h7_CALM_RandomForest_N12_t0", "algo": "RandomForest", "regime": "CALM", "horizon": 7, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "LOW_Lowes_ret_20d", "Industrial_Production_zscore_60d", "EXC_Exelon_ret_1d", "TED_Spread_zscore_60d", "AMD_ret_5d", "AXP_Amex_ret_20d", "PLD_Prologis_ret_5d", "EQR_Equity_ret_1d", "IYM_BasicMaterials_ret_20d", "EFFR_ret_1d"], "is_new": true}, {"model_id": "new_h7_CALM_RandomForest_N12_t1", "algo": "RandomForest", "regime": "CALM", "horizon": 7, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "BA_ret_1d", "US3Y_Rate_ret_5d", "US3M_Rate_zscore_60d", "SPY_zscore_60d", "EMR_Emerson_ret_20d", "EWC_Canada_zscore_60d", "DHR_ret_1d", "CLX_Clorox_vol_20d", "LMT_LockheedMartin_ret_1d", "EWG_Germany_vol_20d"], "is_new": true}, {"model_id": "new_h7_CALM_RandomForest_N12_t2", "algo": "RandomForest", "regime": "CALM", "horizon": 7, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWS_Singapore_ret_5d", "DOW_Price_zscore_60d", "AXP_Amex_vol_20d", "CI_Cigna_vol_20d", "SLB_Schlumberger_ret_1d", "NOC_Northrop_ret_20d", "CPB_CampbellSoup_zscore_60d", "MSTR_Bitcoin3_ret_1d", "EOG_EOGResources_ret_5d", "MS_MorganStanley_zscore_60d"], "is_new": true}, {"model_id": "new_h7_CALM_RandomForest_N12_t3", "algo": "RandomForest", "regime": "CALM", "horizon": 7, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "BA_ret_1d", "VOD_Vodafone_zscore_60d", "SO_SouthernCo_ret_5d", "AVB_AvalonBay_zscore_60d", "PFE_ret_1d", "T10Y2Y_Spread_ret_5d", "ORCL_zscore_60d", "DAX_Germany_zscore_60d", "hmm_p_stress", "CLX_Clorox_vol_20d"], "is_new": true}, {"model_id": "new_h7_CALM_RandomForest_N12_t4", "algo": "RandomForest", "regime": "CALM", "horizon": 7, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "heston_ev_h3", "BLK_BlackRock_zscore_60d", "HangSeng_HK_vol_20d", "BA_ret_1d", "EQR_Equity_ret_1d", "ENB_EnbridgeInc_ret_1d", "SBUX_vol_20d", "FedFunds_zscore_60d", "XLY_Disc_vol_20d", "EWC_Canada_zscore_60d"], "is_new": true}, {"model_id": "new_h7_CALM_RandomForest_N12_t5", "algo": "RandomForest", "regime": "CALM", "horizon": 7, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "XLV_Health_zscore_60d", "MO_AltriaMG_ret_1d", "EWM_Malaysia_ret_1d", "MSTR_Bitcoin3_ret_20d", "hmm_p_stress", "CCI_CrownCastle_vol_20d", "LLY_zscore_60d", "PG_ret_20d", "heston_var_ev_h5", "EWQ_France_ret_20d"], "is_new": true}, {"model_id": "new_h7_CALM_RandomForest_N12_t6", "algo": "RandomForest", "regime": "CALM", "horizon": 7, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "PAYX_Paychex_zscore_60d", "PAYX_Paychex_ret_20d", "DAX_Germany_zscore_60d", "spx_abs_ret_max_5d", "NOC_Northrop_ret_20d", "3M_ret_5d", "EWG_Germany_vol_20d", "EWJ_Japan_vol_20d", "EWA_Australia_ret_1d", "MSTR_Bitcoin3_ret_1d"], "is_new": true}, {"model_id": "new_h7_CALM_RandomForest_N12_t7", "algo": "RandomForest", "regime": "CALM", "horizon": 7, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "US5Y_Rate_ret_5d", "US3M_Rate_vol_20d", "XOM_ret_20d", "VRP_ma5", "XLB_Materials_zscore_60d", "AORD_AUS_zscore_60d", "MSTR_Bitcoin3_ret_5d", "ORCL_zscore_60d", "heston_var_ev_h3", "AMD_ret_5d"], "is_new": true}, {"model_id": "new_h7_CALM_RandomForest_N15_t0", "algo": "RandomForest", "regime": "CALM", "horizon": 7, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "IYM_BasicMaterials_ret_20d", "AXP_Amex_ret_20d", "TED_Spread_vol_20d", "CPB_CampbellSoup_ret_5d", "AMGN_Amgen_ret_1d", "ITT_ITTInc_ret_5d", "EWS_Singapore_ret_5d", "DE_Deere_vol_20d", "CPB_CampbellSoup_vol_20d", "ORCL_vol_20d", "EQIX_Equinix_ret_5d", "DAX_Germany_zscore_60d", "US5Y_Rate_ret_5d"], "is_new": true}, {"model_id": "new_h7_CALM_RandomForest_N15_t1", "algo": "RandomForest", "regime": "CALM", "horizon": 7, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "CPB_CampbellSoup_vol_20d", "CTAS_Cintas_vol_20d", "US3Y_Rate_ret_5d", "TED_Spread_zscore_60d", "NVDA_vol_20d", "IBEX_Spain_ret_20d", "GE_ret_1d", "QQQ_vol_20d", "CI_Cigna_vol_20d", "EWA_Australia_zscore_60d", "EWM_Malaysia_vol_20d", "HD_ret_1d", "SPY_zscore_60d"], "is_new": true}, {"model_id": "new_h7_CALM_RandomForest_N15_t2", "algo": "RandomForest", "regime": "CALM", "horizon": 7, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "TM_Telephone_vol_20d", "M_Macys_vol_20d", "BA_ret_1d", "MSTR_Bitcoin3_ret_1d", "EQR_Equity_ret_1d", "IWM_SmallCap_vol_20d", "EWJ_Japan_vol_20d", "NOC_Northrop_ret_20d", "Brent_Oil_FRED_ret_20d", "EWG_Germany_vol_20d", "spx_vol_5d", "SPY_zscore_60d", "CPB_CampbellSoup_ret_20d"], "is_new": true}, {"model_id": "new_h7_CALM_RandomForest_N15_t3", "algo": "RandomForest", "regime": "CALM", "horizon": 7, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "SJM_JM_Smucker_ret_1d", "PPL_PPL_ret_1d", "PG_ret_20d", "Brent_Oil_FRED_ret_20d", "Brent_Oil_FRED_ret_5d", "AORD_AUS_zscore_60d", "US6M_Rate_ret_20d", "AXP_Amex_ret_20d", "US7Y_Rate_ret_20d", "EWS_Singapore_ret_5d", "EWC_Canada_zscore_60d", "spx_abs_ret_max_5d", "EMR_Emerson_ret_20d"], "is_new": true}, {"model_id": "new_h7_CALM_RandomForest_N15_t4", "algo": "RandomForest", "regime": "CALM", "horizon": 7, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "LUV_SouthwestAir_ret_5d", "SBUX_ret_5d", "EWJ_Japan_vol_20d", "CLX_Clorox_vol_20d", "MSTR_Bitcoin3_ret_5d", "XLB_Materials_zscore_60d", "MO_AltriaMG_ret_1d", "Nikkei_Japan_zscore_60d", "DOW_Price_zscore_60d", "US1Y_Rate_ret_20d", "ITT_ITTInc_ret_5d", "US3M_Rate_vol_20d", "spx_vol_5d"], "is_new": true}, {"model_id": "new_h7_CALM_RandomForest_N15_t5", "algo": "RandomForest", "regime": "CALM", "horizon": 7, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "LOW_Lowes_ret_20d", "TXN_vol_20d", "spx_abs_ret_max_5d", "DOW_Price_zscore_60d", "AMZN_ret_5d", "XLY_Disc_vol_20d", "CMCSA_ret_1d", "AMGN_Amgen_ret_1d", "AMD_ret_1d", "EWQ_France_ret_20d", "Nikkei_Japan_vol_20d", "Nikkei_Japan_zscore_60d", "PLD_Prologis_ret_5d"], "is_new": true}, {"model_id": "new_h7_CALM_RandomForest_N15_t6", "algo": "RandomForest", "regime": "CALM", "horizon": 7, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWM_Malaysia_vol_20d", "EWL_Switzerland_zscore_60d", "SBUX_zscore_60d", "PCAR_PaccarInc_ret_5d", "VVIX_ret_20d", "XLK_Tech_zscore_60d", "PAYX_Paychex_zscore_60d", "heston_var_ev_h3", "XLV_Health_zscore_60d", "EWG_Germany_vol_20d", "Brent_Oil_FRED_ret_20d", "LLY_zscore_60d", "T10Y2Y_Spread_ret_5d"], "is_new": true}, {"model_id": "new_h7_CALM_RandomForest_N15_t7", "algo": "RandomForest", "regime": "CALM", "horizon": 7, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "PPL_PPL_ret_1d", "HD_ret_20d", "heston_var_ev_h5", "SPY_zscore_60d", "Industrial_Production_zscore_60d", "SO_SouthernCo_ret_5d", "Michigan_Sentiment_ret_20d", "T10Y2Y_Spread_ret_5d", "TM_Telephone_vol_20d", "EWM_Malaysia_vol_20d", "spx_abs_ret_max_5d", "ES_Evergy_ret_1d", "LUV_SouthwestAir_ret_5d"], "is_new": true}, {"model_id": "new_h7_CALM_RandomForest_N20_t0", "algo": "RandomForest", "regime": "CALM", "horizon": 7, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWL_Switzerland_zscore_60d", "TM_Telephone_ret_1d", "PLD_Prologis_ret_5d", "EFFR_ret_1d", "EWA_Australia_ret_1d", "EWY_Korea_zscore_60d", "CPB_CampbellSoup_vol_20d", "DIS_vol_20d", "VRP_ma5", "EWL_Switzerland_vol_20d", "US5Y_Rate_ret_5d", "XLV_Health_zscore_60d", "vix_mean_abs_ret_5d", "CPB_CampbellSoup_zscore_60d", "CCI_CrownCastle_vol_20d", "US3Y_Rate_ret_5d", "NVDA_vol_20d", "SJM_JM_Smucker_ret_5d"], "is_new": true}, {"model_id": "new_h7_CALM_RandomForest_N20_t1", "algo": "RandomForest", "regime": "CALM", "horizon": 7, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "CPB_CampbellSoup_ret_20d", "EWQ_France_ret_20d", "SBUX_zscore_60d", "LUV_SouthwestAir_ret_5d", "XLB_Materials_zscore_60d", "Nikkei_Japan_vol_20d", "HD_ret_20d", "SJM_JM_Smucker_ret_5d", "TM_Telephone_vol_20d", "PAYX_Paychex_zscore_60d", "EOG_EOGResources_vol_20d", "ASX_Australia_ret_5d", "MSTR_Bitcoin3_ret_1d", "SJM_JM_Smucker_ret_1d", "XLF_Fin_vol_20d", "PLD_Prologis_ret_5d", "spx_momentum_3d", "ENB_EnbridgeInc_ret_1d"], "is_new": true}, {"model_id": "new_h7_CALM_RandomForest_N20_t2", "algo": "RandomForest", "regime": "CALM", "horizon": 7, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "SO_SouthernCo_ret_5d", "EOG_EOGResources_vol_20d", "MS_MorganStanley_zscore_60d", "VOD_Vodafone_zscore_60d", "heston_var_ev_h7", "LUV_SouthwestAir_ret_5d", "HD_ret_5d", "US6M_Rate_ret_20d", "AMGN_Amgen_ret_1d", "BLK_BlackRock_zscore_60d", "Brent_Oil_FRED_ret_5d", "AORD_AUS_zscore_60d", "EWM_Malaysia_ret_1d", "TED_Spread_vol_20d", "AMD_ret_5d", "CPB_CampbellSoup_ret_5d", "EQIX_Equinix_ret_5d", "CPB_CampbellSoup_ret_20d"], "is_new": true}, {"model_id": "new_h7_CALM_RandomForest_N20_t3", "algo": "RandomForest", "regime": "CALM", "horizon": 7, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "Michigan_Sentiment_ret_20d", "EWH_HongKong_ret_5d", "EWG_Germany_ret_20d", "BLK_BlackRock_zscore_60d", "EOG_EOGResources_ret_5d", "US3Y_Rate_ret_5d", "PAYX_Paychex_vol_20d", "EWM_Malaysia_vol_20d", "US6M_Rate_ret_20d", "JNJ_ret_1d", "VOD_Vodafone_zscore_60d", "US5Y_Rate_ret_5d", "XOM_ret_20d", "EFFR_ret_1d", "SBUX_zscore_60d", "HangSeng_HK_vol_20d", "AXP_Amex_ret_20d", "XLF_Fin_vol_20d"], "is_new": true}, {"model_id": "new_h7_CALM_RandomForest_N20_t4", "algo": "RandomForest", "regime": "CALM", "horizon": 7, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "TED_Spread_vol_20d", "EWH_HongKong_ret_5d", "XLF_Fin_vol_20d", "AVB_AvalonBay_zscore_60d", "NFCI_ret_5d", "TGT_Target_zscore_60d", "EXC_Exelon_zscore_60d", "BDX_Becton_Dickinson_ret_20d", "INTC_ret_5d", "CMCSA_ret_1d", "IWM_SmallCap_vol_20d", "MSTR_Bitcoin3_ret_5d", "EWG_Germany_vol_20d", "US3Y_Rate_ret_5d", "EWL_Switzerland_zscore_60d", "LUV_SouthwestAir_ret_5d", "DAX_Germany_vol_20d", "EWQ_France_ret_20d"], "is_new": true}, {"model_id": "new_h7_CALM_RandomForest_N20_t5", "algo": "RandomForest", "regime": "CALM", "horizon": 7, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "AVB_AvalonBay_zscore_60d", "BTI_BritishAmerican_ret_5d", "EWA_Australia_ret_1d", "spx_abs_ret_max_5d", "AXP_Amex_vol_20d", "EWG_Germany_vol_20d", "IBEX_Spain_ret_20d", "DOW_Price_zscore_60d", "EOG_EOGResources_ret_5d", "TM_Telephone_vol_20d", "US3Y_Rate_ret_5d", "US5Y_Rate_ret_5d", "IYR_US_REIT2_zscore_60d", "CPB_CampbellSoup_ret_20d", "US1Y_Rate_ret_20d", "spx_vol_5d", "EMR_Emerson_ret_20d", "HD_zscore_60d"], "is_new": true}, {"model_id": "new_h7_CALM_RandomForest_N20_t6", "algo": "RandomForest", "regime": "CALM", "horizon": 7, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "Industrial_Production_zscore_60d", "AMD_ret_5d", "US5Y_Rate_ret_5d", "HD_ret_5d", "T_ret_1d", "HUM_Humana_ret_5d", "NEE_NextEra_ret_20d", "vix_acceleration_1d", "LOW_Lowes_ret_5d", "NVDA_vol_20d", "VRP_ma5", "3M_vol_20d", "EWJ_Japan_vol_20d", "EWA_Australia_zscore_60d", "M_Macys_vol_20d", "LMT_LockheedMartin_ret_1d", "HD_ret_1d", "INTC_ret_5d"], "is_new": true}, {"model_id": "new_h7_CALM_RandomForest_N20_t7", "algo": "RandomForest", "regime": "CALM", "horizon": 7, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "HangSeng_HK_vol_20d", "EWS_Singapore_ret_5d", "HangSeng_HK_ret_1d", "PLD_Prologis_ret_5d", "VRP_ma5", "Retail_Sales_zscore_60d", "MO_AltriaMG_ret_1d", "NVDA_vol_20d", "HUM_Humana_ret_5d", "heston_ev_h3", "EFFR_ret_1d", "QQQ_vol_20d", "EWG_Germany_vol_20d", "EWC_Canada_zscore_60d", "US7Y_Rate_ret_20d", "XLV_Health_zscore_60d", "WTI_Oil_FRED_zscore_60d", "MSTR_Bitcoin3_ret_20d"], "is_new": true}, {"model_id": "new_h7_CALM_RandomForest_N25_t0", "algo": "RandomForest", "regime": "CALM", "horizon": 7, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "BLK_BlackRock_zscore_60d", "heston_var_ev_h7", "NWL_Newell_ret_20d", "Core_PCE_zscore_60d", "CMCSA_ret_1d", "EFFR_ret_1d", "spx_momentum_3d", "PCAR_PaccarInc_ret_5d", "AMD_ret_1d", "GE_ret_1d", "EWJ_Japan_vol_20d", "QQQ_vol_20d", "US3Y_Rate_ret_5d", "NVDA_vol_20d", "LMT_LockheedMartin_ret_1d", "AMZN_ret_5d", "US1Y_Rate_ret_5d", "HD_ret_20d", "EWG_Germany_ret_20d", "EWA_Australia_ret_1d", "PG_ret_20d", "DE_Deere_ret_5d", "EWY_Korea_zscore_60d"], "is_new": true}, {"model_id": "new_h7_CALM_RandomForest_N25_t1", "algo": "RandomForest", "regime": "CALM", "horizon": 7, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "XLV_Health_zscore_60d", "ORCL_zscore_60d", "T_ret_1d", "EMR_Emerson_ret_20d", "SBUX_ret_5d", "NFCI_ret_5d", "XLF_Fin_vol_20d", "TXN_vol_20d", "CPB_CampbellSoup_ret_5d", "spx_abs_ret_max_5d", "CTAS_Cintas_vol_20d", "NVDA_vol_20d", "HD_ret_5d", "DIS_vol_20d", "BA_ret_1d", "SBUX_vol_20d", "EWY_Korea_zscore_60d", "EWM_Malaysia_ret_1d", "PLD_Prologis_ret_5d", "INTC_ret_1d", "Brent_Oil_FRED_ret_20d", "PFE_ret_1d", "QQQ_vol_20d"], "is_new": true}, {"model_id": "new_h7_CALM_RandomForest_N25_t2", "algo": "RandomForest", "regime": "CALM", "horizon": 7, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "Core_PCE_zscore_60d", "AXP_Amex_ret_20d", "US30Y_Rate_ret_20d", "CLX_Clorox_vol_20d", "Nikkei_Japan_zscore_60d", "Industrial_Production_zscore_60d", "SJM_JM_Smucker_ret_5d", "TXN_vol_20d", "SBUX_ret_5d", "spx_vol_5d", "PAYX_Paychex_ret_20d", "Brent_Oil_FRED_ret_5d", "BTI_BritishAmerican_ret_5d", "US3Y_Rate_ret_5d", "MS_MorganStanley_ret_5d", "Retail_Sales_zscore_60d", "EXC_Exelon_zscore_60d", "SBUX_zscore_60d", "PCAR_PaccarInc_ret_5d", "JNJ_ret_1d", "EWL_Switzerland_vol_20d", "CCI_CrownCastle_vol_20d", "MSTR_Bitcoin3_ret_5d"], "is_new": true}, {"model_id": "new_h7_CALM_RandomForest_N25_t3", "algo": "RandomForest", "regime": "CALM", "horizon": 7, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "Nikkei_Japan_zscore_60d", "MSTR_Bitcoin3_ret_20d", "US3M_Rate_zscore_60d", "XLV_Health_zscore_60d", "VVIX_ret_20d", "MO_AltriaMG_ret_1d", "LMT_LockheedMartin_ret_1d", "CI_Cigna_vol_20d", "Industrial_Production_zscore_60d", "HD_ret_5d", "BLK_BlackRock_zscore_60d", "EWY_Korea_ret_20d", "DOW_Price_zscore_60d", "US1Y_Rate_ret_20d", "XOM_ret_1d", "ORCL_vol_20d", "spx_momentum_3d", "HD_zscore_60d", "PCAR_PaccarInc_ret_5d", "IYR_US_REIT2_zscore_60d", "EWQ_France_zscore_60d", "NEE_NextEra_ret_20d", "NFCI_ret_5d"], "is_new": true}, {"model_id": "new_h7_CALM_RandomForest_N25_t4", "algo": "RandomForest", "regime": "CALM", "horizon": 7, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "NEE_NextEra_ret_20d", "CPB_CampbellSoup_ret_5d", "HD_ret_5d", "LUV_SouthwestAir_ret_5d", "INTC_ret_5d", "GE_ret_1d", "MRK_Merck_zscore_60d", "Brent_Oil_FRED_ret_20d", "CTAS_Cintas_vol_20d", "SBUX_vol_20d", "JNJ_ret_1d", "spx_abs_ret_max_5d", "US6M_Rate_ret_20d", "MO_AltriaMG_ret_1d", "SO_SouthernCo_ret_5d", "EWL_Switzerland_vol_20d", "US3Y_Rate_ret_5d", "TED_Spread_zscore_60d", "DOW_Price_zscore_60d", "CMCSA_ret_1d", "LMT_LockheedMartin_ret_1d", "ORCL_zscore_60d", "AMGN_Amgen_ret_1d"], "is_new": true}, {"model_id": "new_h7_CALM_RandomForest_N25_t5", "algo": "RandomForest", "regime": "CALM", "horizon": 7, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "MSTR_Bitcoin3_ret_20d", "DHR_ret_1d", "XLF_Fin_vol_20d", "LMT_LockheedMartin_vol_20d", "Michigan_Sentiment_ret_20d", "ORCL_zscore_60d", "ES_Evergy_ret_1d", "EWJ_Japan_vol_20d", "3M_ret_5d", "SLB_Schlumberger_ret_1d", "IWM_SmallCap_vol_20d", "ITT_ITTInc_ret_5d", "GILD_Gilead_ret_20d", "HUM_Humana_ret_5d", "EWL_Switzerland_zscore_60d", "NFCI_ret_5d", "HangSeng_HK_ret_1d", "HD_ret_1d", "US7Y_Rate_ret_20d", "EWM_Malaysia_ret_1d", "Retail_Sales_zscore_60d", "HangSeng_HK_vol_20d", "SBUX_ret_5d"], "is_new": true}, {"model_id": "new_h7_CALM_RandomForest_N25_t6", "algo": "RandomForest", "regime": "CALM", "horizon": 7, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "TM_Telephone_vol_20d", "T_ret_1d", "EWG_Germany_vol_20d", "SO_SouthernCo_ret_5d", "GE_ret_1d", "US6M_Rate_ret_20d", "CPB_CampbellSoup_ret_5d", "Nikkei_Japan_vol_20d", "hmm_p_stress", "SBUX_vol_20d", "Core_PCE_zscore_60d", "LUV_SouthwestAir_ret_5d", "ORCL_zscore_60d", "MS_MorganStanley_zscore_60d", "DHR_vol_20d", "US30Y_Rate_ret_20d", "VRP_ma5", "HangSeng_HK_ret_5d", "US1Y_Rate_ret_5d", "AORD_AUS_zscore_60d", "XLY_Disc_vol_20d", "EWY_Korea_ret_20d", "EWL_Switzerland_vol_20d"], "is_new": true}, {"model_id": "new_h7_CALM_RandomForest_N25_t7", "algo": "RandomForest", "regime": "CALM", "horizon": 7, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "Michigan_Sentiment_ret_20d", "EXC_Exelon_zscore_60d", "PLD_Prologis_ret_5d", "SBUX_zscore_60d", "US30Y_Rate_ret_20d", "T_ret_1d", "BA_ret_1d", "US5Y_Rate_ret_5d", "EWG_Germany_vol_20d", "ORCL_vol_20d", "PCAR_PaccarInc_ret_5d", "MSTR_Bitcoin3_ret_20d", "NOC_Northrop_ret_20d", "CPB_CampbellSoup_vol_20d", "PFE_ret_1d", "gjr_condvar_h1", "EOG_EOGResources_vol_20d", "BLK_BlackRock_zscore_60d", "AMZN_ret_5d", "ENB_EnbridgeInc_ret_1d", "EWM_Malaysia_vol_20d", "CTAS_Cintas_vol_20d", "EWQ_France_ret_20d"], "is_new": true}, {"model_id": "new_h7_CALM_RandomForest_N30_t0", "algo": "RandomForest", "regime": "CALM", "horizon": 7, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "AMT_AmericanTower_ret_1d", "CMCSA_ret_1d", "HD_ret_1d", "INTC_ret_1d", "ASX_Australia_ret_5d", "Brent_Oil_FRED_ret_5d", "Nikkei_Japan_zscore_60d", "DE_Deere_vol_20d", "HD_ret_20d", "HangSeng_HK_ret_1d", "HangSeng_HK_vol_20d", "3M_ret_5d", "AMZN_ret_5d", "XLY_Disc_vol_20d", "spx_abs_ret_max_5d", "EWM_Malaysia_vol_20d", "PLD_Prologis_ret_5d", "DAX_Germany_zscore_60d", "DE_Deere_ret_5d", "HangSeng_HK_ret_5d", "PFE_ret_1d", "EMR_Emerson_ret_20d", "VVIX_ret_20d", "DOW_Price_zscore_60d", "US7Y_Rate_ret_20d", "EWM_Malaysia_ret_1d", "BA_ret_1d", "LMT_LockheedMartin_vol_20d"], "is_new": true}, {"model_id": "new_h7_CALM_RandomForest_N30_t1", "algo": "RandomForest", "regime": "CALM", "horizon": 7, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "heston_var_ev_h7", "TM_Telephone_ret_1d", "BLK_BlackRock_zscore_60d", "US30Y_Rate_ret_20d", "DHR_vol_20d", "EWG_Germany_ret_20d", "BDX_Becton_Dickinson_ret_20d", "AORD_AUS_zscore_60d", "M_Macys_vol_20d", "PLD_Prologis_ret_5d", "US3M_Rate_zscore_60d", "XLB_Materials_zscore_60d", "EOG_EOGResources_vol_20d", "SJM_JM_Smucker_ret_5d", "HD_ret_5d", "MS_MorganStanley_ret_5d", "hmm_p_stress", "US6M_Rate_ret_20d", "IYM_BasicMaterials_ret_20d", "Industrial_Production_zscore_60d", "JNJ_ret_1d", "MSTR_Bitcoin3_ret_20d", "EWY_Korea_zscore_60d", "XOM_ret_1d", "DIS_vol_20d", "XLK_Tech_zscore_60d", "3M_ret_5d", "TGT_Target_zscore_60d"], "is_new": true}, {"model_id": "new_h7_CALM_RandomForest_N30_t2", "algo": "RandomForest", "regime": "CALM", "horizon": 7, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "3M_ret_5d", "Core_CPI_zscore_60d", "spx_momentum_3d", "CMCSA_ret_1d", "US3Y_Rate_ret_5d", "3M_vol_20d", "SLB_Schlumberger_ret_1d", "AVB_AvalonBay_zscore_60d", "HD_ret_1d", "PFE_ret_1d", "WTI_Oil_FRED_zscore_60d", "HangSeng_HK_ret_1d", "US6M_Rate_ret_20d", "AXP_Amex_ret_20d", "LMT_LockheedMartin_vol_20d", "CLX_Clorox_vol_20d", "MRK_Merck_zscore_60d", "T10Y2Y_Spread_ret_5d", "HD_ret_20d", "SPY_zscore_60d", "XLB_Materials_zscore_60d", "heston_var_ev_h7", "DE_Deere_ret_5d", "XLF_Fin_vol_20d", "VOD_Vodafone_zscore_60d", "BTI_BritishAmerican_ret_5d", "ES_Evergy_ret_1d", "BDX_Becton_Dickinson_ret_20d"], "is_new": true}, {"model_id": "new_h7_CALM_RandomForest_N30_t3", "algo": "RandomForest", "regime": "CALM", "horizon": 7, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "IWM_SmallCap_vol_20d", "SLB_Schlumberger_ret_1d", "DAX_Germany_vol_20d", "heston_var_ev_h5", "Brent_Oil_FRED_ret_20d", "GILD_Gilead_ret_20d", "CPB_CampbellSoup_zscore_60d", "Nikkei_Japan_zscore_60d", "SO_SouthernCo_ret_5d", "US3M_Rate_vol_20d", "JNJ_ret_1d", "ASX_Australia_vol_20d", "Michigan_Sentiment_ret_20d", "LMT_LockheedMartin_ret_1d", "PAYX_Paychex_ret_20d", "EWA_Australia_ret_1d", "heston_var_ev_h7", "EWM_Malaysia_vol_20d", "EOG_EOGResources_vol_20d", "TM_Telephone_ret_1d", "Industrial_Production_zscore_60d", "CLX_Clorox_vol_20d", "IYM_BasicMaterials_ret_20d", "MRK_Merck_zscore_60d", "DHR_ret_1d", "heston_ev_h3", "PLD_Prologis_ret_5d", "FedFunds_zscore_60d"], "is_new": true}, {"model_id": "new_h7_CALM_RandomForest_N30_t4", "algo": "RandomForest", "regime": "CALM", "horizon": 7, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "QQQ_vol_20d", "PLD_Prologis_ret_5d", "Core_CPI_zscore_60d", "heston_var_ev_h5", "XLB_Materials_zscore_60d", "AXP_Amex_ret_20d", "EOG_EOGResources_vol_20d", "EWA_Australia_zscore_60d", "EWA_Australia_ret_1d", "MSTR_Bitcoin3_ret_1d", "US30Y_Rate_ret_20d", "DOW_Price_zscore_60d", "US1Y_Rate_ret_20d", "PAYX_Paychex_vol_20d", "TED_Spread_zscore_60d", "BLK_BlackRock_zscore_60d", "T_ret_1d", "US3M_Rate_zscore_60d", "PFE_ret_1d", "AMZN_ret_5d", "SJM_JM_Smucker_ret_1d", "US7Y_Rate_ret_20d", "IBEX_Spain_ret_20d", "SBUX_zscore_60d", "EFFR_vol_20d", "BA_ret_1d", "JNJ_ret_1d", "DE_Deere_vol_20d"], "is_new": true}, {"model_id": "new_h7_CALM_RandomForest_N30_t5", "algo": "RandomForest", "regime": "CALM", "horizon": 7, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "LMT_LockheedMartin_vol_20d", "heston_var_ev_h5", "ASX_Australia_ret_5d", "TM_Telephone_vol_20d", "MSTR_Bitcoin3_ret_5d", "EWM_Malaysia_vol_20d", "XLY_Disc_vol_20d", "DHR_vol_20d", "SJM_JM_Smucker_ret_1d", "PFE_ret_1d", "MS_MorganStanley_zscore_60d", "VRP_ma5", "US30Y_Rate_ret_20d", "EWY_Korea_zscore_60d", "EWA_Australia_ret_1d", "DAX_Germany_vol_20d", "CPB_CampbellSoup_zscore_60d", "ENB_EnbridgeInc_ret_1d", "HD_ret_5d", "AXP_Amex_ret_20d", "spx_momentum_3d", "MS_MorganStanley_ret_1d", "3M_vol_20d", "heston_var_ev_h7", "EWS_Singapore_ret_5d", "M_Macys_vol_20d", "NOC_Northrop_ret_20d", "TED_Spread_zscore_60d"], "is_new": true}, {"model_id": "new_h7_CALM_RandomForest_N30_t6", "algo": "RandomForest", "regime": "CALM", "horizon": 7, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "HD_ret_20d", "Core_CPI_zscore_60d", "CPB_CampbellSoup_vol_20d", "SPY_zscore_60d", "MO_AltriaMG_ret_1d", "EWM_Malaysia_vol_20d", "IWM_SmallCap_vol_20d", "NWL_Newell_ret_20d", "EWY_Korea_zscore_60d", "DAX_Germany_zscore_60d", "hmm_p_stress", "Brent_Oil_FRED_ret_20d", "MS_MorganStanley_ret_5d", "EXC_Exelon_ret_1d", "EQIX_Equinix_ret_5d", "DAX_Germany_vol_20d", "ORCL_vol_20d", "EWS_Singapore_ret_5d", "HangSeng_HK_vol_20d", "MS_MorganStanley_zscore_60d", "CCI_CrownCastle_vol_20d", "AMD_ret_1d", "TXN_vol_20d", "PAYX_Paychex_vol_20d", "INTC_ret_5d", "US3M_Rate_vol_20d", "T10Y2Y_Spread_ret_5d", "XOM_ret_1d"], "is_new": true}, {"model_id": "new_h7_CALM_RandomForest_N30_t7", "algo": "RandomForest", "regime": "CALM", "horizon": 7, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "TM_Telephone_vol_20d", "CI_Cigna_vol_20d", "ENB_EnbridgeInc_ret_1d", "HD_ret_20d", "ASX_Australia_vol_20d", "ITT_ITTInc_ret_5d", "Retail_Sales_zscore_60d", "US7Y_Rate_ret_20d", "EFFR_ret_1d", "MSTR_Bitcoin3_ret_5d", "US1Y_Rate_ret_20d", "DHR_ret_1d", "GD_GeneralDynamics_zscore_60d", "SLB_Schlumberger_ret_5d", "EWQ_France_ret_20d", "PPL_PPL_ret_1d", "IYR_US_REIT2_zscore_60d", "CLX_Clorox_vol_20d", "MS_MorganStanley_zscore_60d", "BA_ret_1d", "MS_MorganStanley_ret_5d", "BTI_BritishAmerican_ret_20d", "HD_ret_5d", "AXP_Amex_ret_20d", "EWJ_Japan_vol_20d", "IWM_SmallCap_vol_20d", "CCI_CrownCastle_vol_20d", "LMT_LockheedMartin_ret_1d"], "is_new": true}, {"model_id": "new_h7_CALM_LogisticRegression_N5_t0", "algo": "LogisticRegression", "regime": "CALM", "horizon": 7, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "XLY_Disc_vol_20d", "QQQ_vol_20d", "HD_zscore_60d"], "is_new": true}, {"model_id": "new_h7_CALM_LogisticRegression_N5_t1", "algo": "LogisticRegression", "regime": "CALM", "horizon": 7, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "T10Y2Y_Spread_ret_5d", "EWM_Malaysia_zscore_60d", "HangSeng_HK_ret_1d"], "is_new": true}, {"model_id": "new_h7_CALM_LogisticRegression_N5_t2", "algo": "LogisticRegression", "regime": "CALM", "horizon": 7, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "IBEX_Spain_ret_20d", "MO_AltriaMG_ret_1d", "AMGN_Amgen_ret_1d"], "is_new": true}, {"model_id": "new_h7_CALM_LogisticRegression_N5_t3", "algo": "LogisticRegression", "regime": "CALM", "horizon": 7, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWJ_Japan_vol_20d", "HangSeng_HK_ret_1d", "VVIX_ret_20d"], "is_new": true}, {"model_id": "new_h7_CALM_LogisticRegression_N5_t4", "algo": "LogisticRegression", "regime": "CALM", "horizon": 7, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "GILD_Gilead_ret_20d", "AMD_ret_5d", "CPB_CampbellSoup_vol_20d"], "is_new": true}, {"model_id": "new_h7_CALM_LogisticRegression_N5_t5", "algo": "LogisticRegression", "regime": "CALM", "horizon": 7, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "NVDA_vol_20d", "TED_Spread_zscore_60d", "HD_ret_5d"], "is_new": true}, {"model_id": "new_h7_CALM_LogisticRegression_N5_t6", "algo": "LogisticRegression", "regime": "CALM", "horizon": 7, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EFFR_ret_1d", "TXN_vol_20d", "ENB_EnbridgeInc_ret_1d"], "is_new": true}, {"model_id": "new_h7_CALM_LogisticRegression_N5_t7", "algo": "LogisticRegression", "regime": "CALM", "horizon": 7, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "DHR_vol_20d", "EWM_Malaysia_ret_1d", "NFCI_ret_5d"], "is_new": true}, {"model_id": "new_h7_CALM_LogisticRegression_N8_t0", "algo": "LogisticRegression", "regime": "CALM", "horizon": 7, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "heston_var_ev_h7", "IYM_BasicMaterials_ret_20d", "CI_Cigna_vol_20d", "CPB_CampbellSoup_ret_20d", "GILD_Gilead_ret_20d", "ASX_Australia_ret_5d"], "is_new": true}, {"model_id": "new_h7_CALM_LogisticRegression_N8_t1", "algo": "LogisticRegression", "regime": "CALM", "horizon": 7, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "3M_vol_20d", "MS_MorganStanley_ret_1d", "QQQ_vol_20d", "VVIX_ret_20d", "HD_ret_20d", "PLD_Prologis_ret_5d"], "is_new": true}, {"model_id": "new_h7_CALM_LogisticRegression_N8_t2", "algo": "LogisticRegression", "regime": "CALM", "horizon": 7, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "ASX_Australia_ret_5d", "spx_vol_5d", "CPB_CampbellSoup_zscore_60d", "BTI_BritishAmerican_ret_5d", "ORCL_vol_20d", "DIS_vol_20d"], "is_new": true}, {"model_id": "new_h7_CALM_LogisticRegression_N8_t3", "algo": "LogisticRegression", "regime": "CALM", "horizon": 7, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "MRK_Merck_zscore_60d", "MS_MorganStanley_ret_5d", "HD_ret_1d", "Core_CPI_zscore_60d", "VVIX_ret_20d", "US7Y_Rate_ret_20d"], "is_new": true}, {"model_id": "new_h7_CALM_LogisticRegression_N8_t4", "algo": "LogisticRegression", "regime": "CALM", "horizon": 7, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "AMD_ret_1d", "XLV_Health_zscore_60d", "Retail_Sales_zscore_60d", "ASX_Australia_vol_20d", "HangSeng_HK_ret_1d", "XLK_Tech_zscore_60d"], "is_new": true}, {"model_id": "new_h7_CALM_LogisticRegression_N8_t5", "algo": "LogisticRegression", "regime": "CALM", "horizon": 7, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "IYM_BasicMaterials_ret_20d", "NFCI_ret_5d", "BTI_BritishAmerican_ret_5d", "spx_momentum_3d", "EMR_Emerson_ret_20d", "CPB_CampbellSoup_ret_5d"], "is_new": true}, {"model_id": "new_h7_CALM_LogisticRegression_N8_t6", "algo": "LogisticRegression", "regime": "CALM", "horizon": 7, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "LOW_Lowes_ret_5d", "EWY_Korea_zscore_60d", "SBUX_vol_20d", "vix_mean_abs_ret_5d", "BLK_BlackRock_zscore_60d", "ORCL_zscore_60d"], "is_new": true}, {"model_id": "new_h7_CALM_LogisticRegression_N8_t7", "algo": "LogisticRegression", "regime": "CALM", "horizon": 7, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "GE_ret_1d", "PCAR_PaccarInc_ret_5d", "DAX_Germany_zscore_60d", "vix_acceleration_1d", "MRK_Merck_zscore_60d", "CPB_CampbellSoup_ret_5d"], "is_new": true}, {"model_id": "new_h7_CALM_LogisticRegression_N10_t0", "algo": "LogisticRegression", "regime": "CALM", "horizon": 7, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "T_ret_1d", "NEE_NextEra_ret_20d", "XOM_ret_20d", "LOW_Lowes_ret_5d", "ITT_ITTInc_ret_5d", "3M_vol_20d", "US7Y_Rate_ret_20d", "US3M_Rate_zscore_60d"], "is_new": true}, {"model_id": "new_h7_CALM_LogisticRegression_N10_t1", "algo": "LogisticRegression", "regime": "CALM", "horizon": 7, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "XLV_Health_zscore_60d", "HangSeng_HK_ret_5d", "EWJ_Japan_vol_20d", "3M_ret_5d", "MO_AltriaMG_ret_1d", "XOM_ret_20d", "TM_Telephone_vol_20d", "PAYX_Paychex_ret_20d"], "is_new": true}, {"model_id": "new_h7_CALM_LogisticRegression_N10_t2", "algo": "LogisticRegression", "regime": "CALM", "horizon": 7, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EXC_Exelon_zscore_60d", "Core_PCE_zscore_60d", "INTC_ret_5d", "CPB_CampbellSoup_zscore_60d", "EWM_Malaysia_vol_20d", "MS_MorganStanley_ret_1d", "CPB_CampbellSoup_ret_20d", "EFFR_ret_1d"], "is_new": true}, {"model_id": "new_h7_CALM_LogisticRegression_N10_t3", "algo": "LogisticRegression", "regime": "CALM", "horizon": 7, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "NWL_Newell_ret_20d", "AMZN_ret_5d", "DAX_Germany_vol_20d", "ES_Evergy_ret_1d", "PPL_PPL_ret_1d", "heston_var_ev_h7", "TED_Spread_vol_20d", "EQR_Equity_ret_1d"], "is_new": true}, {"model_id": "new_h7_CALM_LogisticRegression_N10_t4", "algo": "LogisticRegression", "regime": "CALM", "horizon": 7, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "Nikkei_Japan_vol_20d", "AXP_Amex_ret_20d", "ENB_EnbridgeInc_ret_1d", "EWY_Korea_zscore_60d", "AMD_ret_1d", "DE_Deere_ret_5d", "Michigan_Sentiment_ret_20d", "EWQ_France_ret_20d"], "is_new": true}, {"model_id": "new_h7_CALM_LogisticRegression_N10_t5", "algo": "LogisticRegression", "regime": "CALM", "horizon": 7, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "VRP_ma5", "MRK_Merck_zscore_60d", "JNJ_ret_1d", "EFFR_vol_20d", "US3Y_Rate_ret_5d", "VOD_Vodafone_zscore_60d", "CPB_CampbellSoup_ret_20d", "ASX_Australia_ret_5d"], "is_new": true}, {"model_id": "new_h7_CALM_LogisticRegression_N10_t6", "algo": "LogisticRegression", "regime": "CALM", "horizon": 7, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "NWL_Newell_ret_20d", "WTI_Oil_FRED_zscore_60d", "Nikkei_Japan_zscore_60d", "SBUX_ret_5d", "AVB_AvalonBay_zscore_60d", "TGT_Target_zscore_60d", "EWA_Australia_zscore_60d", "DHR_vol_20d"], "is_new": true}, {"model_id": "new_h7_CALM_LogisticRegression_N10_t7", "algo": "LogisticRegression", "regime": "CALM", "horizon": 7, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "3M_vol_20d", "T_ret_1d", "EWA_Australia_zscore_60d", "HUM_Humana_ret_5d", "ASX_Australia_ret_5d", "AXP_Amex_vol_20d", "CI_Cigna_vol_20d", "PPL_PPL_ret_1d"], "is_new": true}, {"model_id": "new_h7_CALM_LogisticRegression_N12_t0", "algo": "LogisticRegression", "regime": "CALM", "horizon": 7, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "HD_ret_5d", "AVB_AvalonBay_zscore_60d", "AMD_ret_5d", "EWY_Korea_ret_20d", "hmm_p_stress", "US30Y_Rate_ret_20d", "CMCSA_ret_1d", "MSTR_Bitcoin3_ret_1d", "AXP_Amex_vol_20d", "AMZN_ret_5d"], "is_new": true}, {"model_id": "new_h7_CALM_LogisticRegression_N12_t1", "algo": "LogisticRegression", "regime": "CALM", "horizon": 7, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "ES_Evergy_ret_1d", "EWY_Korea_zscore_60d", "EXC_Exelon_zscore_60d", "EWA_Australia_zscore_60d", "EWM_Malaysia_ret_1d", "PAYX_Paychex_zscore_60d", "HangSeng_HK_vol_20d", "US3M_Rate_zscore_60d", "LLY_zscore_60d", "NVDA_vol_20d"], "is_new": true}, {"model_id": "new_h7_CALM_LogisticRegression_N12_t2", "algo": "LogisticRegression", "regime": "CALM", "horizon": 7, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWM_Malaysia_ret_1d", "EXC_Exelon_ret_1d", "SBUX_ret_5d", "AXP_Amex_ret_20d", "PAYX_Paychex_ret_20d", "DHR_ret_1d", "HD_ret_20d", "Nikkei_Japan_zscore_60d", "EQR_Equity_ret_1d", "IYM_BasicMaterials_ret_20d"], "is_new": true}, {"model_id": "new_h7_CALM_LogisticRegression_N12_t3", "algo": "LogisticRegression", "regime": "CALM", "horizon": 7, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "DE_Deere_ret_5d", "BLK_BlackRock_zscore_60d", "Industrial_Production_zscore_60d", "spx_vol_5d", "IYR_US_REIT2_zscore_60d", "AORD_AUS_zscore_60d", "MSTR_Bitcoin3_ret_1d", "SBUX_ret_5d", "DHR_ret_1d", "SLB_Schlumberger_ret_5d"], "is_new": true}, {"model_id": "new_h7_CALM_LogisticRegression_N12_t4", "algo": "LogisticRegression", "regime": "CALM", "horizon": 7, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "T10Y2Y_Spread_ret_5d", "EWQ_France_ret_20d", "Industrial_Production_zscore_60d", "EOG_EOGResources_ret_5d", "EWA_Australia_ret_1d", "EWJ_Japan_vol_20d", "VRP_ma5", "CCI_CrownCastle_vol_20d", "VVIX_ret_20d", "XLB_Materials_zscore_60d"], "is_new": true}, {"model_id": "new_h7_CALM_LogisticRegression_N12_t5", "algo": "LogisticRegression", "regime": "CALM", "horizon": 7, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "Retail_Sales_zscore_60d", "AORD_AUS_zscore_60d", "DHR_ret_1d", "US1Y_Rate_ret_5d", "SCHW_Schwab_ret_5d", "EXC_Exelon_ret_1d", "T_ret_1d", "US7Y_Rate_ret_20d", "EOG_EOGResources_vol_20d", "TM_Telephone_ret_1d"], "is_new": true}, {"model_id": "new_h7_CALM_LogisticRegression_N12_t6", "algo": "LogisticRegression", "regime": "CALM", "horizon": 7, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWA_Australia_ret_1d", "CCI_CrownCastle_vol_20d", "US7Y_Rate_ret_20d", "PFE_ret_1d", "US3M_Rate_zscore_60d", "XLK_Tech_zscore_60d", "CPB_CampbellSoup_ret_20d", "DHR_ret_1d", "EWQ_France_zscore_60d", "BLK_BlackRock_zscore_60d"], "is_new": true}, {"model_id": "new_h7_CALM_LogisticRegression_N12_t7", "algo": "LogisticRegression", "regime": "CALM", "horizon": 7, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWC_Canada_zscore_60d", "LMT_LockheedMartin_vol_20d", "CPB_CampbellSoup_vol_20d", "PAYX_Paychex_vol_20d", "HangSeng_HK_ret_1d", "US7Y_Rate_ret_20d", "MSTR_Bitcoin3_ret_1d", "HUM_Humana_ret_5d", "MS_MorganStanley_ret_1d", "TM_Telephone_vol_20d"], "is_new": true}, {"model_id": "new_h7_CALM_LogisticRegression_N15_t0", "algo": "LogisticRegression", "regime": "CALM", "horizon": 7, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "XLB_Materials_zscore_60d", "HD_ret_20d", "PCAR_PaccarInc_ret_5d", "CPB_CampbellSoup_vol_20d", "HangSeng_HK_ret_5d", "EWM_Malaysia_vol_20d", "spx_momentum_3d", "spx_vol_5d", "XLK_Tech_zscore_60d", "MRK_Merck_zscore_60d", "CMCSA_ret_1d", "NOC_Northrop_ret_20d", "Core_CPI_zscore_60d"], "is_new": true}, {"model_id": "new_h7_CALM_LogisticRegression_N15_t1", "algo": "LogisticRegression", "regime": "CALM", "horizon": 7, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "3M_vol_20d", "INTC_ret_1d", "EWS_Singapore_ret_5d", "US5Y_Rate_ret_5d", "EWY_Korea_zscore_60d", "PFE_ret_1d", "PAYX_Paychex_zscore_60d", "DIS_vol_20d", "heston_var_ev_h3", "EWA_Australia_zscore_60d", "BTI_BritishAmerican_ret_20d", "EWM_Malaysia_ret_1d", "EWM_Malaysia_vol_20d"], "is_new": true}, {"model_id": "new_h7_CALM_LogisticRegression_N15_t2", "algo": "LogisticRegression", "regime": "CALM", "horizon": 7, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "HD_ret_20d", "EWL_Switzerland_vol_20d", "HD_zscore_60d", "heston_var_ev_h5", "INTC_ret_5d", "Nikkei_Japan_vol_20d", "HangSeng_HK_ret_5d", "TGT_Target_zscore_60d", "MSTR_Bitcoin3_ret_5d", "SCHW_Schwab_ret_5d", "Brent_Oil_FRED_ret_5d", "heston_var_ev_h7", "CMCSA_ret_1d"], "is_new": true}, {"model_id": "new_h7_CALM_LogisticRegression_N15_t3", "algo": "LogisticRegression", "regime": "CALM", "horizon": 7, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWM_Malaysia_ret_1d", "AMD_ret_5d", "EWG_Germany_vol_20d", "EWH_HongKong_ret_5d", "EXC_Exelon_ret_1d", "BTI_BritishAmerican_ret_20d", "MSTR_Bitcoin3_ret_20d", "AMD_ret_1d", "HangSeng_HK_ret_5d", "MSTR_Bitcoin3_ret_5d", "ENB_EnbridgeInc_ret_1d", "US7Y_Rate_ret_20d", "AXP_Amex_vol_20d"], "is_new": true}, {"model_id": "new_h7_CALM_LogisticRegression_N15_t4", "algo": "LogisticRegression", "regime": "CALM", "horizon": 7, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "ASX_Australia_vol_20d", "Brent_Oil_FRED_ret_5d", "HD_ret_1d", "DHR_ret_1d", "Retail_Sales_zscore_60d", "INTC_ret_1d", "LUV_SouthwestAir_ret_5d", "heston_var_ev_h5", "EOG_EOGResources_ret_5d", "Michigan_Sentiment_ret_20d", "EWQ_France_ret_20d", "EXC_Exelon_zscore_60d", "BA_ret_1d"], "is_new": true}, {"model_id": "new_h7_CALM_LogisticRegression_N15_t5", "algo": "LogisticRegression", "regime": "CALM", "horizon": 7, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWY_Korea_ret_20d", "NFCI_ret_5d", "EOG_EOGResources_ret_5d", "EXC_Exelon_zscore_60d", "WTI_Oil_FRED_zscore_60d", "EWM_Malaysia_ret_1d", "TM_Telephone_vol_20d", "XOM_ret_20d", "spx_momentum_3d", "GE_ret_1d", "IYM_BasicMaterials_ret_20d", "PG_ret_20d", "CLX_Clorox_vol_20d"], "is_new": true}, {"model_id": "new_h7_CALM_LogisticRegression_N15_t6", "algo": "LogisticRegression", "regime": "CALM", "horizon": 7, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "HD_zscore_60d", "VOD_Vodafone_zscore_60d", "T_ret_1d", "DE_Deere_ret_5d", "JNJ_ret_1d", "BDX_Becton_Dickinson_ret_20d", "EXC_Exelon_zscore_60d", "NEE_NextEra_ret_20d", "CCI_CrownCastle_vol_20d", "M_Macys_vol_20d", "CPB_CampbellSoup_zscore_60d", "Retail_Sales_zscore_60d", "Brent_Oil_FRED_ret_5d"], "is_new": true}, {"model_id": "new_h7_CALM_LogisticRegression_N15_t7", "algo": "LogisticRegression", "regime": "CALM", "horizon": 7, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "GILD_Gilead_ret_20d", "SBUX_zscore_60d", "CPB_CampbellSoup_zscore_60d", "CI_Cigna_vol_20d", "TXN_vol_20d", "INTC_ret_5d", "MS_MorganStanley_ret_1d", "EWQ_France_zscore_60d", "AMD_ret_5d", "NVDA_vol_20d", "US3M_Rate_vol_20d", "AXP_Amex_vol_20d", "NEE_NextEra_ret_20d"], "is_new": true}, {"model_id": "new_h7_CALM_LogisticRegression_N20_t0", "algo": "LogisticRegression", "regime": "CALM", "horizon": 7, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "CPB_CampbellSoup_ret_20d", "EQR_Equity_ret_1d", "Core_CPI_zscore_60d", "DAX_Germany_zscore_60d", "US6M_Rate_ret_20d", "US1Y_Rate_ret_5d", "SPY_zscore_60d", "PFE_ret_1d", "EXC_Exelon_ret_1d", "NOC_Northrop_ret_20d", "heston_var_ev_h3", "MSTR_Bitcoin3_ret_20d", "EWQ_France_ret_20d", "ITT_ITTInc_ret_5d", "CPB_CampbellSoup_vol_20d", "CCI_CrownCastle_vol_20d", "DE_Deere_vol_20d", "EWA_Australia_ret_1d"], "is_new": true}, {"model_id": "new_h7_CALM_LogisticRegression_N20_t1", "algo": "LogisticRegression", "regime": "CALM", "horizon": 7, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "US5Y_Rate_ret_5d", "DE_Deere_vol_20d", "MS_MorganStanley_ret_5d", "EWG_Germany_vol_20d", "VVIX_ret_20d", "EWA_Australia_zscore_60d", "ASX_Australia_vol_20d", "US6M_Rate_ret_20d", "Nikkei_Japan_zscore_60d", "spx_abs_ret_max_5d", "LLY_zscore_60d", "T10Y2Y_Spread_ret_5d", "heston_var_ev_h7", "DHR_vol_20d", "DOW_Price_zscore_60d", "US30Y_Rate_ret_20d", "EWM_Malaysia_vol_20d", "3M_vol_20d"], "is_new": true}, {"model_id": "new_h7_CALM_LogisticRegression_N20_t2", "algo": "LogisticRegression", "regime": "CALM", "horizon": 7, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "HangSeng_HK_ret_5d", "TED_Spread_zscore_60d", "CCI_CrownCastle_vol_20d", "US1Y_Rate_ret_20d", "NOC_Northrop_ret_20d", "US7Y_Rate_ret_20d", "LOW_Lowes_ret_20d", "INTC_ret_5d", "US6M_Rate_ret_20d", "hmm_p_stress", "spx_vol_5d", "NWL_Newell_ret_20d", "CLX_Clorox_vol_20d", "ORCL_vol_20d", "VRP_ma5", "heston_ev_h3", "GD_GeneralDynamics_zscore_60d", "US3M_Rate_zscore_60d"], "is_new": true}, {"model_id": "new_h7_CALM_LogisticRegression_N20_t3", "algo": "LogisticRegression", "regime": "CALM", "horizon": 7, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "AMZN_ret_5d", "EWG_Germany_ret_20d", "XOM_ret_20d", "AMD_ret_5d", "US1Y_Rate_ret_20d", "EOG_EOGResources_vol_20d", "PFE_ret_1d", "EWL_Switzerland_vol_20d", "Nikkei_Japan_vol_20d", "spx_abs_ret_max_5d", "MSTR_Bitcoin3_ret_20d", "Industrial_Production_zscore_60d", "NFCI_ret_5d", "PPL_PPL_ret_1d", "heston_ev_h3", "MRK_Merck_zscore_60d", "Michigan_Sentiment_ret_20d", "VOD_Vodafone_zscore_60d"], "is_new": true}, {"model_id": "new_h7_CALM_LogisticRegression_N20_t4", "algo": "LogisticRegression", "regime": "CALM", "horizon": 7, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "ASX_Australia_ret_5d", "VRP_ma5", "IWM_SmallCap_vol_20d", "US1Y_Rate_ret_20d", "EFFR_ret_1d", "EMR_Emerson_ret_20d", "PAYX_Paychex_zscore_60d", "hmm_p_stress", "INTC_ret_5d", "3M_vol_20d", "heston_var_ev_h5", "HangSeng_HK_ret_5d", "HD_ret_5d", "NVDA_vol_20d", "ITT_ITTInc_ret_5d", "MSTR_Bitcoin3_ret_20d", "CTAS_Cintas_vol_20d", "NOC_Northrop_ret_20d"], "is_new": true}, {"model_id": "new_h7_CALM_LogisticRegression_N20_t5", "algo": "LogisticRegression", "regime": "CALM", "horizon": 7, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "Michigan_Sentiment_ret_20d", "LOW_Lowes_ret_20d", "T10Y2Y_Spread_ret_5d", "HangSeng_HK_ret_1d", "VRP_ma5", "BTI_BritishAmerican_ret_20d", "HD_ret_1d", "HD_zscore_60d", "CTAS_Cintas_vol_20d", "MSTR_Bitcoin3_ret_20d", "TED_Spread_vol_20d", "AMGN_Amgen_ret_1d", "LLY_zscore_60d", "EQR_Equity_ret_1d", "LMT_LockheedMartin_vol_20d", "ORCL_zscore_60d", "EWL_Switzerland_zscore_60d", "heston_var_ev_h5"], "is_new": true}, {"model_id": "new_h7_CALM_LogisticRegression_N20_t6", "algo": "LogisticRegression", "regime": "CALM", "horizon": 7, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWL_Switzerland_vol_20d", "AVB_AvalonBay_zscore_60d", "SPY_zscore_60d", "EWG_Germany_vol_20d", "SLB_Schlumberger_ret_1d", "XOM_ret_1d", "DHR_vol_20d", "EXC_Exelon_zscore_60d", "GD_GeneralDynamics_zscore_60d", "US6M_Rate_ret_20d", "BLK_BlackRock_zscore_60d", "AMD_ret_1d", "WTI_Oil_FRED_zscore_60d", "BTI_BritishAmerican_ret_20d", "INTC_ret_5d", "EXC_Exelon_ret_1d", "EFFR_vol_20d", "EMR_Emerson_ret_20d"], "is_new": true}, {"model_id": "new_h7_CALM_LogisticRegression_N20_t7", "algo": "LogisticRegression", "regime": "CALM", "horizon": 7, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWH_HongKong_ret_5d", "3M_vol_20d", "PPL_PPL_ret_1d", "GILD_Gilead_ret_20d", "SJM_JM_Smucker_ret_5d", "PG_ret_20d", "HD_ret_5d", "LMT_LockheedMartin_ret_1d", "EWA_Australia_ret_1d", "GD_GeneralDynamics_zscore_60d", "spx_vol_5d", "heston_var_ev_h3", "DAX_Germany_vol_20d", "BDX_Becton_Dickinson_ret_20d", "vix_mean_abs_ret_5d", "PLD_Prologis_ret_5d", "hmm_p_stress", "HD_zscore_60d"], "is_new": true}, {"model_id": "new_h7_CALM_LogisticRegression_N25_t0", "algo": "LogisticRegression", "regime": "CALM", "horizon": 7, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "SCHW_Schwab_ret_5d", "EWA_Australia_ret_1d", "spx_momentum_3d", "MRK_Merck_zscore_60d", "US6M_Rate_ret_20d", "PFE_ret_1d", "XOM_ret_20d", "EMR_Emerson_ret_20d", "TM_Telephone_vol_20d", "BLK_BlackRock_zscore_60d", "HangSeng_HK_ret_5d", "MS_MorganStanley_ret_5d", "NEE_NextEra_ret_20d", "PAYX_Paychex_ret_20d", "HUM_Humana_ret_5d", "NOC_Northrop_ret_20d", "XLV_Health_zscore_60d", "XLB_Materials_zscore_60d", "EWY_Korea_ret_20d", "DE_Deere_vol_20d", "LOW_Lowes_ret_20d", "CPB_CampbellSoup_zscore_60d", "MSTR_Bitcoin3_ret_1d"], "is_new": true}, {"model_id": "new_h7_CALM_LogisticRegression_N25_t1", "algo": "LogisticRegression", "regime": "CALM", "horizon": 7, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "SCHW_Schwab_ret_5d", "hmm_p_stress", "AORD_AUS_zscore_60d", "Brent_Oil_FRED_ret_20d", "SO_SouthernCo_ret_5d", "XLV_Health_zscore_60d", "PLD_Prologis_ret_5d", "EFFR_vol_20d", "EWM_Malaysia_ret_1d", "XLF_Fin_vol_20d", "SJM_JM_Smucker_ret_5d", "IYM_BasicMaterials_ret_20d", "SPY_zscore_60d", "PCAR_PaccarInc_ret_5d", "US3M_Rate_vol_20d", "spx_abs_ret_max_5d", "AXP_Amex_vol_20d", "M_Macys_vol_20d", "ORCL_vol_20d", "NEE_NextEra_ret_20d", "LOW_Lowes_ret_20d", "INTC_ret_5d", "DIS_vol_20d"], "is_new": true}, {"model_id": "new_h7_CALM_LogisticRegression_N25_t2", "algo": "LogisticRegression", "regime": "CALM", "horizon": 7, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWY_Korea_ret_20d", "HD_ret_20d", "Nikkei_Japan_vol_20d", "US1Y_Rate_ret_20d", "spx_vol_5d", "PAYX_Paychex_zscore_60d", "EWG_Germany_vol_20d", "AMZN_ret_5d", "EWY_Korea_zscore_60d", "BA_ret_1d", "3M_vol_20d", "heston_ev_h3", "XOM_ret_1d", "MO_AltriaMG_ret_1d", "EWJ_Japan_vol_20d", "MSTR_Bitcoin3_ret_1d", "LMT_LockheedMartin_ret_1d", "ORCL_vol_20d", "PPL_PPL_ret_1d", "EWG_Germany_ret_20d", "SJM_JM_Smucker_ret_1d", "ES_Evergy_ret_1d", "EQR_Equity_ret_1d"], "is_new": true}, {"model_id": "new_h7_CALM_LogisticRegression_N25_t3", "algo": "LogisticRegression", "regime": "CALM", "horizon": 7, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "DHR_ret_1d", "SO_SouthernCo_ret_5d", "TM_Telephone_vol_20d", "GD_GeneralDynamics_zscore_60d", "GE_ret_1d", "CTAS_Cintas_vol_20d", "IBEX_Spain_ret_20d", "DAX_Germany_zscore_60d", "MO_AltriaMG_ret_1d", "EWM_Malaysia_zscore_60d", "heston_ev_h3", "NVDA_vol_20d", "AMGN_Amgen_ret_1d", "EOG_EOGResources_ret_5d", "US1Y_Rate_ret_20d", "Nikkei_Japan_zscore_60d", "XLB_Materials_zscore_60d", "AMT_AmericanTower_ret_1d", "US1Y_Rate_ret_5d", "SLB_Schlumberger_ret_5d", "SJM_JM_Smucker_ret_1d", "Retail_Sales_zscore_60d", "INTC_ret_1d"], "is_new": true}, {"model_id": "new_h7_CALM_LogisticRegression_N25_t4", "algo": "LogisticRegression", "regime": "CALM", "horizon": 7, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "VRP_ma5", "GE_ret_1d", "HD_zscore_60d", "NFCI_ret_5d", "EWQ_France_zscore_60d", "vix_acceleration_1d", "EOG_EOGResources_vol_20d", "SJM_JM_Smucker_ret_5d", "MO_AltriaMG_ret_1d", "EWM_Malaysia_vol_20d", "LMT_LockheedMartin_ret_1d", "PPL_PPL_ret_1d", "IBEX_Spain_ret_20d", "Nikkei_Japan_vol_20d", "EFFR_ret_1d", "AXP_Amex_vol_20d", "XLY_Disc_vol_20d", "NEE_NextEra_ret_20d", "JNJ_ret_1d", "XLV_Health_zscore_60d", "SBUX_zscore_60d", "EWM_Malaysia_ret_1d", "IYM_BasicMaterials_ret_20d"], "is_new": true}, {"model_id": "new_h7_CALM_LogisticRegression_N25_t5", "algo": "LogisticRegression", "regime": "CALM", "horizon": 7, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EFFR_ret_1d", "TED_Spread_zscore_60d", "US3M_Rate_vol_20d", "SLB_Schlumberger_ret_5d", "AXP_Amex_ret_20d", "BA_ret_1d", "CCI_CrownCastle_vol_20d", "SJM_JM_Smucker_ret_1d", "M_Macys_vol_20d", "US6M_Rate_ret_20d", "MS_MorganStanley_ret_1d", "MS_MorganStanley_ret_5d", "Nikkei_Japan_vol_20d", "heston_var_ev_h5", "DAX_Germany_vol_20d", "ITT_ITTInc_ret_5d", "NEE_NextEra_ret_20d", "EWG_Germany_ret_20d", "AMGN_Amgen_ret_1d", "ORCL_zscore_60d", "LUV_SouthwestAir_ret_5d", "US3Y_Rate_ret_5d", "DHR_ret_1d"], "is_new": true}, {"model_id": "new_h7_CALM_LogisticRegression_N25_t6", "algo": "LogisticRegression", "regime": "CALM", "horizon": 7, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "heston_var_ev_h7", "MSTR_Bitcoin3_ret_5d", "MSTR_Bitcoin3_ret_20d", "US30Y_Rate_ret_20d", "AXP_Amex_vol_20d", "ASX_Australia_ret_5d", "AMGN_Amgen_ret_1d", "EOG_EOGResources_ret_5d", "PAYX_Paychex_vol_20d", "EWH_HongKong_ret_5d", "CPB_CampbellSoup_ret_20d", "EWQ_France_zscore_60d", "PLD_Prologis_ret_5d", "DHR_ret_1d", "NVDA_vol_20d", "IYM_BasicMaterials_ret_20d", "Brent_Oil_FRED_ret_20d", "DE_Deere_vol_20d", "ITT_ITTInc_ret_5d", "INTC_ret_1d", "spx_momentum_3d", "JNJ_ret_1d", "PPL_PPL_ret_1d"], "is_new": true}, {"model_id": "new_h7_CALM_LogisticRegression_N25_t7", "algo": "LogisticRegression", "regime": "CALM", "horizon": 7, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWY_Korea_ret_20d", "FedFunds_zscore_60d", "gjr_condvar_h1", "AXP_Amex_vol_20d", "EQR_Equity_ret_1d", "CI_Cigna_vol_20d", "Core_PCE_zscore_60d", "XLB_Materials_zscore_60d", "XLK_Tech_zscore_60d", "BLK_BlackRock_zscore_60d", "M_Macys_vol_20d", "HD_ret_5d", "EWH_HongKong_ret_5d", "Nikkei_Japan_vol_20d", "ORCL_zscore_60d", "PFE_ret_1d", "US5Y_Rate_ret_5d", "HD_ret_20d", "EXC_Exelon_ret_1d", "PCAR_PaccarInc_ret_5d", "PLD_Prologis_ret_5d", "XOM_ret_20d", "EMR_Emerson_ret_20d"], "is_new": true}, {"model_id": "new_h7_CALM_LogisticRegression_N30_t0", "algo": "LogisticRegression", "regime": "CALM", "horizon": 7, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWJ_Japan_vol_20d", "CPB_CampbellSoup_vol_20d", "US30Y_Rate_ret_20d", "EOG_EOGResources_vol_20d", "SBUX_ret_5d", "EWQ_France_ret_20d", "MSTR_Bitcoin3_ret_1d", "US6M_Rate_ret_20d", "spx_momentum_3d", "CPB_CampbellSoup_zscore_60d", "XLF_Fin_vol_20d", "CPB_CampbellSoup_ret_5d", "Nikkei_Japan_vol_20d", "Retail_Sales_zscore_60d", "PG_ret_20d", "CI_Cigna_vol_20d", "AORD_AUS_zscore_60d", "TED_Spread_vol_20d", "MS_MorganStanley_zscore_60d", "MS_MorganStanley_ret_5d", "US5Y_Rate_ret_5d", "GILD_Gilead_ret_20d", "PAYX_Paychex_vol_20d", "EWY_Korea_ret_20d", "US1Y_Rate_ret_20d", "BDX_Becton_Dickinson_ret_20d", "DAX_Germany_zscore_60d", "US1Y_Rate_ret_5d"], "is_new": true}, {"model_id": "new_h7_CALM_LogisticRegression_N30_t1", "algo": "LogisticRegression", "regime": "CALM", "horizon": 7, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "PPL_PPL_ret_1d", "TED_Spread_vol_20d", "US6M_Rate_ret_20d", "LMT_LockheedMartin_vol_20d", "EOG_EOGResources_vol_20d", "vix_mean_abs_ret_5d", "SO_SouthernCo_ret_5d", "Nikkei_Japan_zscore_60d", "US5Y_Rate_ret_5d", "3M_ret_5d", "EWC_Canada_zscore_60d", "AORD_AUS_zscore_60d", "AMZN_ret_5d", "IBEX_Spain_ret_20d", "LLY_zscore_60d", "CPB_CampbellSoup_ret_20d", "EOG_EOGResources_ret_5d", "VOD_Vodafone_zscore_60d", "PAYX_Paychex_ret_20d", "VVIX_ret_20d", "CPB_CampbellSoup_vol_20d", "NOC_Northrop_ret_20d", "PFE_ret_1d", "CMCSA_ret_1d", "MS_MorganStanley_ret_1d", "Brent_Oil_FRED_ret_5d", "EWY_Korea_ret_20d", "SBUX_ret_5d"], "is_new": true}, {"model_id": "new_h7_CALM_LogisticRegression_N30_t2", "algo": "LogisticRegression", "regime": "CALM", "horizon": 7, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "AMT_AmericanTower_ret_1d", "PAYX_Paychex_ret_20d", "EWG_Germany_ret_20d", "LMT_LockheedMartin_ret_1d", "spx_momentum_3d", "EOG_EOGResources_ret_5d", "SCHW_Schwab_ret_5d", "SO_SouthernCo_ret_5d", "EXC_Exelon_ret_1d", "MO_AltriaMG_ret_1d", "Core_PCE_zscore_60d", "DHR_vol_20d", "vix_acceleration_1d", "BTI_BritishAmerican_ret_20d", "3M_ret_5d", "heston_ev_h3", "EWL_Switzerland_vol_20d", "SJM_JM_Smucker_ret_5d", "TM_Telephone_ret_1d", "EWG_Germany_vol_20d", "HangSeng_HK_vol_20d", "TXN_vol_20d", "PAYX_Paychex_vol_20d", "PG_ret_20d", "VOD_Vodafone_zscore_60d", "GE_ret_1d", "CPB_CampbellSoup_zscore_60d", "LLY_zscore_60d"], "is_new": true}, {"model_id": "new_h7_CALM_LogisticRegression_N30_t3", "algo": "LogisticRegression", "regime": "CALM", "horizon": 7, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "SLB_Schlumberger_ret_1d", "ORCL_vol_20d", "T_ret_1d", "HangSeng_HK_ret_1d", "PLD_Prologis_ret_5d", "NEE_NextEra_ret_20d", "DAX_Germany_zscore_60d", "Industrial_Production_zscore_60d", "US5Y_Rate_ret_5d", "US3Y_Rate_ret_5d", "M_Macys_vol_20d", "PAYX_Paychex_zscore_60d", "ENB_EnbridgeInc_ret_1d", "EWY_Korea_ret_20d", "EWC_Canada_zscore_60d", "HangSeng_HK_vol_20d", "MSTR_Bitcoin3_ret_5d", "Brent_Oil_FRED_ret_20d", "spx_abs_ret_max_5d", "SLB_Schlumberger_ret_5d", "US3M_Rate_zscore_60d", "CPB_CampbellSoup_ret_20d", "SBUX_zscore_60d", "Retail_Sales_zscore_60d", "CI_Cigna_vol_20d", "BDX_Becton_Dickinson_ret_20d", "EFFR_vol_20d", "BLK_BlackRock_zscore_60d"], "is_new": true}, {"model_id": "new_h7_CALM_LogisticRegression_N30_t4", "algo": "LogisticRegression", "regime": "CALM", "horizon": 7, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "CTAS_Cintas_vol_20d", "DAX_Germany_vol_20d", "TM_Telephone_vol_20d", "XLV_Health_zscore_60d", "HUM_Humana_ret_5d", "PAYX_Paychex_zscore_60d", "GE_ret_1d", "PPL_PPL_ret_1d", "Brent_Oil_FRED_ret_5d", "PFE_ret_1d", "US3M_Rate_vol_20d", "US3Y_Rate_ret_5d", "EWG_Germany_ret_20d", "TED_Spread_vol_20d", "TED_Spread_zscore_60d", "Michigan_Sentiment_ret_20d", "SO_SouthernCo_ret_5d", "EWY_Korea_zscore_60d", "TGT_Target_zscore_60d", "SBUX_vol_20d", "TM_Telephone_ret_1d", "NFCI_ret_5d", "EWY_Korea_ret_20d", "US1Y_Rate_ret_5d", "EMR_Emerson_ret_20d", "heston_var_ev_h7", "MS_MorganStanley_ret_1d", "TXN_vol_20d"], "is_new": true}, {"model_id": "new_h7_CALM_LogisticRegression_N30_t5", "algo": "LogisticRegression", "regime": "CALM", "horizon": 7, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "LMT_LockheedMartin_ret_1d", "BTI_BritishAmerican_ret_5d", "EWM_Malaysia_ret_1d", "CI_Cigna_vol_20d", "EMR_Emerson_ret_20d", "SBUX_vol_20d", "heston_ev_h3", "PPL_PPL_ret_1d", "EWJ_Japan_vol_20d", "CCI_CrownCastle_vol_20d", "IYM_BasicMaterials_ret_20d", "DAX_Germany_zscore_60d", "TM_Telephone_ret_1d", "CLX_Clorox_vol_20d", "LOW_Lowes_ret_20d", "XLF_Fin_vol_20d", "3M_ret_5d", "HD_ret_1d", "EWA_Australia_zscore_60d", "Michigan_Sentiment_ret_20d", "CPB_CampbellSoup_ret_20d", "DHR_ret_1d", "CPB_CampbellSoup_vol_20d", "IWM_SmallCap_vol_20d", "Core_PCE_zscore_60d", "MS_MorganStanley_ret_1d", "heston_var_ev_h3", "HD_ret_5d"], "is_new": true}, {"model_id": "new_h7_CALM_LogisticRegression_N30_t6", "algo": "LogisticRegression", "regime": "CALM", "horizon": 7, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "SPY_zscore_60d", "US3M_Rate_vol_20d", "Brent_Oil_FRED_ret_5d", "EWA_Australia_zscore_60d", "3M_vol_20d", "SJM_JM_Smucker_ret_5d", "WTI_Oil_FRED_zscore_60d", "vix_mean_abs_ret_5d", "CI_Cigna_vol_20d", "SJM_JM_Smucker_ret_1d", "XOM_ret_1d", "CPB_CampbellSoup_ret_5d", "Core_PCE_zscore_60d", "IWM_SmallCap_vol_20d", "XLF_Fin_vol_20d", "SLB_Schlumberger_ret_5d", "PAYX_Paychex_vol_20d", "DHR_vol_20d", "FedFunds_zscore_60d", "TED_Spread_zscore_60d", "EWH_HongKong_ret_5d", "HD_zscore_60d", "EWQ_France_ret_20d", "BA_ret_1d", "NEE_NextEra_ret_20d", "PPL_PPL_ret_1d", "HUM_Humana_ret_5d", "BTI_BritishAmerican_ret_20d"], "is_new": true}, {"model_id": "new_h7_CALM_LogisticRegression_N30_t7", "algo": "LogisticRegression", "regime": "CALM", "horizon": 7, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWQ_France_ret_20d", "VRP_ma5", "TED_Spread_vol_20d", "T_ret_1d", "XLK_Tech_zscore_60d", "CPB_CampbellSoup_vol_20d", "EWJ_Japan_vol_20d", "XLB_Materials_zscore_60d", "XOM_ret_20d", "vix_acceleration_1d", "SJM_JM_Smucker_ret_1d", "spx_abs_ret_max_5d", "MS_MorganStanley_ret_1d", "TED_Spread_zscore_60d", "TGT_Target_zscore_60d", "SBUX_ret_5d", "SO_SouthernCo_ret_5d", "SCHW_Schwab_ret_5d", "EQIX_Equinix_ret_5d", "GILD_Gilead_ret_20d", "EWC_Canada_zscore_60d", "heston_var_ev_h5", "DAX_Germany_zscore_60d", "IBEX_Spain_ret_20d", "heston_var_ev_h7", "ORCL_vol_20d", "PLD_Prologis_ret_5d", "Industrial_Production_zscore_60d"], "is_new": true}, {"model_id": "new_h7_NORMAL_XGBoost_N5_t0", "algo": "XGBoost", "regime": "NORMAL", "horizon": 7, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWQ_France_zscore_60d", "PLD_Prologis_ret_5d", "CPB_CampbellSoup_ret_5d"], "is_new": true}, {"model_id": "new_h7_NORMAL_XGBoost_N5_t1", "algo": "XGBoost", "regime": "NORMAL", "horizon": 7, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "JNJ_ret_1d", "NOC_Northrop_ret_20d", "Brent_Oil_FRED_ret_5d"], "is_new": true}, {"model_id": "new_h7_NORMAL_XGBoost_N5_t2", "algo": "XGBoost", "regime": "NORMAL", "horizon": 7, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "HD_zscore_60d", "SJM_JM_Smucker_ret_1d", "Brent_Oil_FRED_ret_5d"], "is_new": true}, {"model_id": "new_h7_NORMAL_XGBoost_N5_t3", "algo": "XGBoost", "regime": "NORMAL", "horizon": 7, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWH_HongKong_ret_5d", "AXP_Amex_ret_20d", "PCAR_PaccarInc_ret_5d"], "is_new": true}, {"model_id": "new_h7_NORMAL_XGBoost_N5_t4", "algo": "XGBoost", "regime": "NORMAL", "horizon": 7, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "CPB_CampbellSoup_zscore_60d", "MSTR_Bitcoin3_ret_1d", "spx_vol_5d"], "is_new": true}, {"model_id": "new_h7_NORMAL_XGBoost_N5_t5", "algo": "XGBoost", "regime": "NORMAL", "horizon": 7, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "SO_SouthernCo_ret_5d", "FedFunds_zscore_60d", "EWM_Malaysia_zscore_60d"], "is_new": true}, {"model_id": "new_h7_NORMAL_XGBoost_N5_t6", "algo": "XGBoost", "regime": "NORMAL", "horizon": 7, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWL_Switzerland_vol_20d", "LUV_SouthwestAir_ret_5d", "EWM_Malaysia_zscore_60d"], "is_new": true}, {"model_id": "new_h7_NORMAL_XGBoost_N5_t7", "algo": "XGBoost", "regime": "NORMAL", "horizon": 7, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "NEE_NextEra_ret_20d", "heston_ev_h3", "WTI_Oil_FRED_zscore_60d"], "is_new": true}, {"model_id": "new_h7_NORMAL_XGBoost_N8_t0", "algo": "XGBoost", "regime": "NORMAL", "horizon": 7, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "AXP_Amex_vol_20d", "Brent_Oil_FRED_ret_20d", "DE_Deere_ret_5d", "HD_ret_1d", "SBUX_zscore_60d", "Core_PCE_zscore_60d"], "is_new": true}, {"model_id": "new_h7_NORMAL_XGBoost_N8_t1", "algo": "XGBoost", "regime": "NORMAL", "horizon": 7, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "SLB_Schlumberger_ret_5d", "EWQ_France_zscore_60d", "heston_var_ev_h7", "EWL_Switzerland_vol_20d", "EWL_Switzerland_zscore_60d", "CI_Cigna_vol_20d"], "is_new": true}, {"model_id": "new_h7_NORMAL_XGBoost_N8_t2", "algo": "XGBoost", "regime": "NORMAL", "horizon": 7, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "heston_var_ev_h7", "EWY_Korea_zscore_60d", "EXC_Exelon_ret_1d", "Industrial_Production_zscore_60d", "EOG_EOGResources_vol_20d", "SBUX_vol_20d"], "is_new": true}, {"model_id": "new_h7_NORMAL_XGBoost_N8_t3", "algo": "XGBoost", "regime": "NORMAL", "horizon": 7, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EQR_Equity_ret_1d", "spx_abs_ret_max_5d", "HangSeng_HK_ret_5d", "DOW_Price_zscore_60d", "SLB_Schlumberger_ret_5d", "EQIX_Equinix_ret_5d"], "is_new": true}, {"model_id": "new_h7_NORMAL_XGBoost_N8_t4", "algo": "XGBoost", "regime": "NORMAL", "horizon": 7, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWL_Switzerland_zscore_60d", "PAYX_Paychex_vol_20d", "SLB_Schlumberger_ret_5d", "TM_Telephone_vol_20d", "EWY_Korea_ret_20d", "CLX_Clorox_vol_20d"], "is_new": true}, {"model_id": "new_h7_NORMAL_XGBoost_N8_t5", "algo": "XGBoost", "regime": "NORMAL", "horizon": 7, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "SBUX_ret_5d", "CPB_CampbellSoup_vol_20d", "DHR_vol_20d", "heston_ev_h3", "EWG_Germany_vol_20d", "SBUX_vol_20d"], "is_new": true}, {"model_id": "new_h7_NORMAL_XGBoost_N8_t6", "algo": "XGBoost", "regime": "NORMAL", "horizon": 7, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWG_Germany_ret_20d", "US30Y_Rate_ret_20d", "EWA_Australia_zscore_60d", "Nikkei_Japan_vol_20d", "FedFunds_zscore_60d", "vix_acceleration_1d"], "is_new": true}, {"model_id": "new_h7_NORMAL_XGBoost_N8_t7", "algo": "XGBoost", "regime": "NORMAL", "horizon": 7, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWS_Singapore_ret_5d", "gjr_condvar_h1", "VVIX_ret_20d", "LOW_Lowes_ret_5d", "EXC_Exelon_zscore_60d", "T_ret_1d"], "is_new": true}, {"model_id": "new_h7_NORMAL_XGBoost_N10_t0", "algo": "XGBoost", "regime": "NORMAL", "horizon": 7, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "CPB_CampbellSoup_ret_20d", "HD_ret_1d", "US3M_Rate_vol_20d", "SPY_zscore_60d", "QQQ_vol_20d", "AMZN_ret_5d", "SO_SouthernCo_ret_5d", "EMR_Emerson_ret_20d"], "is_new": true}, {"model_id": "new_h7_NORMAL_XGBoost_N10_t1", "algo": "XGBoost", "regime": "NORMAL", "horizon": 7, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "IBEX_Spain_ret_20d", "hmm_p_stress", "LMT_LockheedMartin_ret_1d", "VVIX_ret_20d", "BLK_BlackRock_zscore_60d", "VOD_Vodafone_zscore_60d", "GD_GeneralDynamics_zscore_60d", "PPL_PPL_ret_1d"], "is_new": true}, {"model_id": "new_h7_NORMAL_XGBoost_N10_t2", "algo": "XGBoost", "regime": "NORMAL", "horizon": 7, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "MSTR_Bitcoin3_ret_1d", "EWQ_France_ret_20d", "DHR_vol_20d", "Industrial_Production_zscore_60d", "DIS_vol_20d", "MRK_Merck_zscore_60d", "SLB_Schlumberger_ret_5d", "SJM_JM_Smucker_ret_5d"], "is_new": true}, {"model_id": "new_h7_NORMAL_XGBoost_N10_t3", "algo": "XGBoost", "regime": "NORMAL", "horizon": 7, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "DHR_vol_20d", "spx_momentum_3d", "HD_ret_5d", "BLK_BlackRock_zscore_60d", "EWJ_Japan_vol_20d", "SJM_JM_Smucker_ret_5d", "Core_CPI_zscore_60d", "EWY_Korea_zscore_60d"], "is_new": true}, {"model_id": "new_h7_NORMAL_XGBoost_N10_t4", "algo": "XGBoost", "regime": "NORMAL", "horizon": 7, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWQ_France_ret_20d", "CLX_Clorox_vol_20d", "EWM_Malaysia_vol_20d", "EWG_Germany_ret_20d", "CPB_CampbellSoup_zscore_60d", "spx_vol_5d", "EFFR_ret_1d", "vix_mean_abs_ret_5d"], "is_new": true}, {"model_id": "new_h7_NORMAL_XGBoost_N10_t5", "algo": "XGBoost", "regime": "NORMAL", "horizon": 7, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWQ_France_zscore_60d", "spx_abs_ret_max_5d", "EXC_Exelon_ret_1d", "SJM_JM_Smucker_ret_5d", "XLB_Materials_zscore_60d", "EWQ_France_ret_20d", "NOC_Northrop_ret_20d", "JNJ_ret_1d"], "is_new": true}, {"model_id": "new_h7_NORMAL_XGBoost_N10_t6", "algo": "XGBoost", "regime": "NORMAL", "horizon": 7, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "PPL_PPL_ret_1d", "3M_ret_5d", "CCI_CrownCastle_vol_20d", "spx_vol_5d", "PG_ret_20d", "ENB_EnbridgeInc_ret_1d", "SLB_Schlumberger_ret_5d", "LMT_LockheedMartin_vol_20d"], "is_new": true}, {"model_id": "new_h7_NORMAL_XGBoost_N10_t7", "algo": "XGBoost", "regime": "NORMAL", "horizon": 7, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "SJM_JM_Smucker_ret_1d", "EWJ_Japan_vol_20d", "Nikkei_Japan_zscore_60d", "GD_GeneralDynamics_zscore_60d", "Core_PCE_zscore_60d", "EWY_Korea_zscore_60d", "WTI_Oil_FRED_zscore_60d", "DHR_ret_1d"], "is_new": true}, {"model_id": "new_h7_NORMAL_XGBoost_N12_t0", "algo": "XGBoost", "regime": "NORMAL", "horizon": 7, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "Core_CPI_zscore_60d", "US30Y_Rate_ret_20d", "CCI_CrownCastle_vol_20d", "AMGN_Amgen_ret_1d", "spx_momentum_3d", "ENB_EnbridgeInc_ret_1d", "US7Y_Rate_ret_20d", "AMZN_ret_5d", "AXP_Amex_vol_20d", "ES_Evergy_ret_1d"], "is_new": true}, {"model_id": "new_h7_NORMAL_XGBoost_N12_t1", "algo": "XGBoost", "regime": "NORMAL", "horizon": 7, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "US3M_Rate_zscore_60d", "NOC_Northrop_ret_20d", "spx_abs_ret_max_5d", "EWS_Singapore_ret_5d", "MSTR_Bitcoin3_ret_1d", "MS_MorganStanley_ret_5d", "XLF_Fin_vol_20d", "3M_ret_5d", "DE_Deere_vol_20d", "3M_vol_20d"], "is_new": true}, {"model_id": "new_h7_NORMAL_XGBoost_N12_t2", "algo": "XGBoost", "regime": "NORMAL", "horizon": 7, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "ES_Evergy_ret_1d", "TED_Spread_zscore_60d", "EMR_Emerson_ret_20d", "CPB_CampbellSoup_vol_20d", "EQIX_Equinix_ret_5d", "LMT_LockheedMartin_vol_20d", "DE_Deere_vol_20d", "US30Y_Rate_ret_20d", "LMT_LockheedMartin_ret_1d", "LOW_Lowes_ret_20d"], "is_new": true}, {"model_id": "new_h7_NORMAL_XGBoost_N12_t3", "algo": "XGBoost", "regime": "NORMAL", "horizon": 7, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWL_Switzerland_vol_20d", "IYR_US_REIT2_zscore_60d", "DE_Deere_vol_20d", "US1Y_Rate_ret_20d", "hmm_p_stress", "EWY_Korea_ret_20d", "EWS_Singapore_ret_5d", "SPY_zscore_60d", "TED_Spread_zscore_60d", "TM_Telephone_vol_20d"], "is_new": true}, {"model_id": "new_h7_NORMAL_XGBoost_N12_t4", "algo": "XGBoost", "regime": "NORMAL", "horizon": 7, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "US30Y_Rate_ret_20d", "NVDA_vol_20d", "Retail_Sales_zscore_60d", "XLV_Health_zscore_60d", "MS_MorganStanley_ret_1d", "EWG_Germany_ret_20d", "BA_ret_1d", "EFFR_vol_20d", "EWM_Malaysia_vol_20d", "LMT_LockheedMartin_vol_20d"], "is_new": true}, {"model_id": "new_h7_NORMAL_XGBoost_N12_t5", "algo": "XGBoost", "regime": "NORMAL", "horizon": 7, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "BDX_Becton_Dickinson_ret_20d", "3M_vol_20d", "AMGN_Amgen_ret_1d", "HD_ret_1d", "vix_acceleration_1d", "MS_MorganStanley_ret_5d", "SPY_zscore_60d", "EWG_Germany_ret_20d", "LOW_Lowes_ret_20d", "DHR_vol_20d"], "is_new": true}, {"model_id": "new_h7_NORMAL_XGBoost_N12_t6", "algo": "XGBoost", "regime": "NORMAL", "horizon": 7, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "BDX_Becton_Dickinson_ret_20d", "PCAR_PaccarInc_ret_5d", "NVDA_vol_20d", "LOW_Lowes_ret_20d", "EFFR_vol_20d", "EWJ_Japan_vol_20d", "EWG_Germany_ret_20d", "INTC_ret_5d", "SBUX_ret_5d", "AXP_Amex_ret_20d"], "is_new": true}, {"model_id": "new_h7_NORMAL_XGBoost_N12_t7", "algo": "XGBoost", "regime": "NORMAL", "horizon": 7, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "INTC_ret_5d", "AXP_Amex_vol_20d", "US3M_Rate_vol_20d", "DHR_ret_1d", "WTI_Oil_FRED_zscore_60d", "T10Y2Y_Spread_ret_5d", "BA_ret_1d", "Michigan_Sentiment_ret_20d", "LMT_LockheedMartin_ret_1d", "CPB_CampbellSoup_ret_5d"], "is_new": true}, {"model_id": "new_h7_NORMAL_XGBoost_N15_t0", "algo": "XGBoost", "regime": "NORMAL", "horizon": 7, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "CPB_CampbellSoup_ret_20d", "spx_abs_ret_max_5d", "DOW_Price_zscore_60d", "SBUX_ret_5d", "GILD_Gilead_ret_20d", "3M_ret_5d", "Core_CPI_zscore_60d", "DIS_vol_20d", "VVIX_ret_20d", "NWL_Newell_ret_20d", "heston_var_ev_h3", "PLD_Prologis_ret_5d", "NOC_Northrop_ret_20d"], "is_new": true}, {"model_id": "new_h7_NORMAL_XGBoost_N15_t1", "algo": "XGBoost", "regime": "NORMAL", "horizon": 7, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "HangSeng_HK_ret_1d", "SBUX_zscore_60d", "NFCI_ret_5d", "SCHW_Schwab_ret_5d", "CI_Cigna_vol_20d", "US7Y_Rate_ret_20d", "DOW_Price_zscore_60d", "DHR_ret_1d", "PCAR_PaccarInc_ret_5d", "TM_Telephone_vol_20d", "ENB_EnbridgeInc_ret_1d", "Core_PCE_zscore_60d", "vix_mean_abs_ret_5d"], "is_new": true}, {"model_id": "new_h7_NORMAL_XGBoost_N15_t2", "algo": "XGBoost", "regime": "NORMAL", "horizon": 7, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "SJM_JM_Smucker_ret_1d", "Michigan_Sentiment_ret_20d", "CPB_CampbellSoup_ret_20d", "DAX_Germany_zscore_60d", "VOD_Vodafone_zscore_60d", "DE_Deere_ret_5d", "Retail_Sales_zscore_60d", "3M_ret_5d", "Nikkei_Japan_vol_20d", "vix_acceleration_1d", "MRK_Merck_zscore_60d", "EWM_Malaysia_ret_1d", "heston_var_ev_h7"], "is_new": true}, {"model_id": "new_h7_NORMAL_XGBoost_N15_t3", "algo": "XGBoost", "regime": "NORMAL", "horizon": 7, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "CI_Cigna_vol_20d", "Core_CPI_zscore_60d", "DHR_ret_1d", "EWL_Switzerland_vol_20d", "PG_ret_20d", "WTI_Oil_FRED_zscore_60d", "EFFR_ret_1d", "SJM_JM_Smucker_ret_5d", "CMCSA_ret_1d", "FedFunds_zscore_60d", "T10Y2Y_Spread_ret_5d", "AXP_Amex_vol_20d", "ENB_EnbridgeInc_ret_1d"], "is_new": true}, {"model_id": "new_h7_NORMAL_XGBoost_N15_t4", "algo": "XGBoost", "regime": "NORMAL", "horizon": 7, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "FedFunds_zscore_60d", "PCAR_PaccarInc_ret_5d", "HangSeng_HK_ret_1d", "GD_GeneralDynamics_zscore_60d", "DAX_Germany_vol_20d", "spx_abs_ret_max_5d", "DAX_Germany_zscore_60d", "HD_zscore_60d", "AMGN_Amgen_ret_1d", "HD_ret_1d", "EWY_Korea_ret_20d", "TGT_Target_zscore_60d", "EXC_Exelon_zscore_60d"], "is_new": true}, {"model_id": "new_h7_NORMAL_XGBoost_N15_t5", "algo": "XGBoost", "regime": "NORMAL", "horizon": 7, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "vix_acceleration_1d", "Brent_Oil_FRED_ret_5d", "CCI_CrownCastle_vol_20d", "LMT_LockheedMartin_vol_20d", "TM_Telephone_vol_20d", "spx_abs_ret_max_5d", "EOG_EOGResources_ret_5d", "QQQ_vol_20d", "AMD_ret_1d", "EFFR_vol_20d", "EXC_Exelon_ret_1d", "gjr_condvar_h1", "LOW_Lowes_ret_5d"], "is_new": true}, {"model_id": "new_h7_NORMAL_XGBoost_N15_t6", "algo": "XGBoost", "regime": "NORMAL", "horizon": 7, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "HD_zscore_60d", "HD_ret_20d", "NVDA_vol_20d", "EWM_Malaysia_vol_20d", "EWJ_Japan_vol_20d", "EOG_EOGResources_ret_5d", "CMCSA_ret_1d", "Brent_Oil_FRED_ret_5d", "XLV_Health_zscore_60d", "EWY_Korea_zscore_60d", "SBUX_ret_5d", "ASX_Australia_ret_5d", "BTI_BritishAmerican_ret_5d"], "is_new": true}, {"model_id": "new_h7_NORMAL_XGBoost_N15_t7", "algo": "XGBoost", "regime": "NORMAL", "horizon": 7, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "US3M_Rate_zscore_60d", "MSTR_Bitcoin3_ret_20d", "EWM_Malaysia_ret_1d", "heston_var_ev_h3", "CI_Cigna_vol_20d", "VRP_ma5", "hmm_p_stress", "AMGN_Amgen_ret_1d", "vix_mean_abs_ret_5d", "CPB_CampbellSoup_zscore_60d", "AVB_AvalonBay_zscore_60d", "PFE_ret_1d", "gjr_condvar_h1"], "is_new": true}, {"model_id": "new_h7_NORMAL_XGBoost_N20_t0", "algo": "XGBoost", "regime": "NORMAL", "horizon": 7, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "SLB_Schlumberger_ret_1d", "MRK_Merck_zscore_60d", "LLY_zscore_60d", "Nikkei_Japan_vol_20d", "PG_ret_20d", "CTAS_Cintas_vol_20d", "EOG_EOGResources_ret_5d", "EWL_Switzerland_vol_20d", "hmm_p_stress", "heston_ev_h3", "AXP_Amex_ret_20d", "Brent_Oil_FRED_ret_5d", "ORCL_vol_20d", "SJM_JM_Smucker_ret_1d", "HangSeng_HK_vol_20d", "EWM_Malaysia_ret_1d", "ASX_Australia_ret_5d", "EFFR_ret_1d"], "is_new": true}, {"model_id": "new_h7_NORMAL_XGBoost_N20_t1", "algo": "XGBoost", "regime": "NORMAL", "horizon": 7, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "Michigan_Sentiment_ret_20d", "EWQ_France_ret_20d", "EOG_EOGResources_ret_5d", "ORCL_vol_20d", "PAYX_Paychex_zscore_60d", "EWM_Malaysia_zscore_60d", "VOD_Vodafone_zscore_60d", "MRK_Merck_zscore_60d", "T10Y2Y_Spread_ret_5d", "XOM_ret_1d", "HD_ret_5d", "SO_SouthernCo_ret_5d", "EWS_Singapore_ret_5d", "EFFR_ret_1d", "MSTR_Bitcoin3_ret_1d", "HangSeng_HK_ret_1d", "NFCI_ret_5d", "GE_ret_1d"], "is_new": true}, {"model_id": "new_h7_NORMAL_XGBoost_N20_t2", "algo": "XGBoost", "regime": "NORMAL", "horizon": 7, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWY_Korea_ret_20d", "AORD_AUS_zscore_60d", "Industrial_Production_zscore_60d", "VRP_ma5", "NFCI_ret_5d", "SLB_Schlumberger_ret_1d", "heston_var_ev_h5", "EWL_Switzerland_zscore_60d", "ENB_EnbridgeInc_ret_1d", "TED_Spread_vol_20d", "DOW_Price_zscore_60d", "EWM_Malaysia_zscore_60d", "BTI_BritishAmerican_ret_20d", "Retail_Sales_zscore_60d", "PPL_PPL_ret_1d", "SCHW_Schwab_ret_5d", "Nikkei_Japan_zscore_60d", "HUM_Humana_ret_5d"], "is_new": true}, {"model_id": "new_h7_NORMAL_XGBoost_N20_t3", "algo": "XGBoost", "regime": "NORMAL", "horizon": 7, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EXC_Exelon_zscore_60d", "MS_MorganStanley_zscore_60d", "T_ret_1d", "CLX_Clorox_vol_20d", "QQQ_vol_20d", "heston_var_ev_h5", "JNJ_ret_1d", "spx_abs_ret_max_5d", "XOM_ret_1d", "INTC_ret_5d", "US1Y_Rate_ret_5d", "SJM_JM_Smucker_ret_1d", "hmm_p_stress", "AXP_Amex_vol_20d", "MSTR_Bitcoin3_ret_20d", "MSTR_Bitcoin3_ret_1d", "SLB_Schlumberger_ret_1d", "BA_ret_1d"], "is_new": true}, {"model_id": "new_h7_NORMAL_XGBoost_N20_t4", "algo": "XGBoost", "regime": "NORMAL", "horizon": 7, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "SBUX_ret_5d", "MO_AltriaMG_ret_1d", "MS_MorganStanley_ret_1d", "NVDA_vol_20d", "EWH_HongKong_ret_5d", "TM_Telephone_ret_1d", "MS_MorganStanley_ret_5d", "EOG_EOGResources_vol_20d", "EWM_Malaysia_ret_1d", "EWA_Australia_zscore_60d", "TGT_Target_zscore_60d", "US1Y_Rate_ret_20d", "GD_GeneralDynamics_zscore_60d", "Brent_Oil_FRED_ret_5d", "AXP_Amex_ret_20d", "spx_vol_5d", "Brent_Oil_FRED_ret_20d", "PLD_Prologis_ret_5d"], "is_new": true}, {"model_id": "new_h7_NORMAL_XGBoost_N20_t5", "algo": "XGBoost", "regime": "NORMAL", "horizon": 7, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "XLF_Fin_vol_20d", "MO_AltriaMG_ret_1d", "NOC_Northrop_ret_20d", "vix_acceleration_1d", "NVDA_vol_20d", "BA_ret_1d", "EWL_Switzerland_vol_20d", "INTC_ret_1d", "Nikkei_Japan_vol_20d", "EQR_Equity_ret_1d", "EWA_Australia_zscore_60d", "T_ret_1d", "AMZN_ret_5d", "EFFR_ret_1d", "Industrial_Production_zscore_60d", "ITT_ITTInc_ret_5d", "MRK_Merck_zscore_60d", "SBUX_ret_5d"], "is_new": true}, {"model_id": "new_h7_NORMAL_XGBoost_N20_t6", "algo": "XGBoost", "regime": "NORMAL", "horizon": 7, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "Michigan_Sentiment_ret_20d", "EWA_Australia_zscore_60d", "DOW_Price_zscore_60d", "vix_acceleration_1d", "XLK_Tech_zscore_60d", "LLY_zscore_60d", "Core_CPI_zscore_60d", "HD_ret_1d", "spx_vol_5d", "NEE_NextEra_ret_20d", "PCAR_PaccarInc_ret_5d", "IBEX_Spain_ret_20d", "BDX_Becton_Dickinson_ret_20d", "hmm_p_stress", "EWG_Germany_vol_20d", "PLD_Prologis_ret_5d", "LUV_SouthwestAir_ret_5d", "M_Macys_vol_20d"], "is_new": true}, {"model_id": "new_h7_NORMAL_XGBoost_N20_t7", "algo": "XGBoost", "regime": "NORMAL", "horizon": 7, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "ES_Evergy_ret_1d", "HD_ret_1d", "EXC_Exelon_zscore_60d", "heston_var_ev_h3", "hmm_p_stress", "Michigan_Sentiment_ret_20d", "DAX_Germany_zscore_60d", "HangSeng_HK_ret_5d", "CMCSA_ret_1d", "Nikkei_Japan_vol_20d", "SJM_JM_Smucker_ret_1d", "GE_ret_1d", "US5Y_Rate_ret_5d", "PG_ret_20d", "GILD_Gilead_ret_20d", "TED_Spread_zscore_60d", "XLB_Materials_zscore_60d", "XLY_Disc_vol_20d"], "is_new": true}, {"model_id": "new_h7_NORMAL_XGBoost_N25_t0", "algo": "XGBoost", "regime": "NORMAL", "horizon": 7, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "FedFunds_zscore_60d", "SLB_Schlumberger_ret_1d", "EWG_Germany_ret_20d", "CTAS_Cintas_vol_20d", "SO_SouthernCo_ret_5d", "DHR_ret_1d", "BTI_BritishAmerican_ret_20d", "CPB_CampbellSoup_zscore_60d", "TXN_vol_20d", "DAX_Germany_vol_20d", "HangSeng_HK_ret_1d", "SPY_zscore_60d", "MRK_Merck_zscore_60d", "ASX_Australia_ret_5d", "NWL_Newell_ret_20d", "CPB_CampbellSoup_vol_20d", "US30Y_Rate_ret_20d", "ORCL_zscore_60d", "US6M_Rate_ret_20d", "US7Y_Rate_ret_20d", "Nikkei_Japan_vol_20d", "EWY_Korea_ret_20d", "EWY_Korea_zscore_60d"], "is_new": true}, {"model_id": "new_h7_NORMAL_XGBoost_N25_t1", "algo": "XGBoost", "regime": "NORMAL", "horizon": 7, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "ASX_Australia_vol_20d", "TED_Spread_vol_20d", "US30Y_Rate_ret_20d", "EQIX_Equinix_ret_5d", "IBEX_Spain_ret_20d", "PPL_PPL_ret_1d", "VRP_ma5", "DE_Deere_ret_5d", "DHR_vol_20d", "ENB_EnbridgeInc_ret_1d", "XLV_Health_zscore_60d", "SCHW_Schwab_ret_5d", "TGT_Target_zscore_60d", "EWG_Germany_vol_20d", "AMT_AmericanTower_ret_1d", "US6M_Rate_ret_20d", "CI_Cigna_vol_20d", "PAYX_Paychex_vol_20d", "US3Y_Rate_ret_5d", "EOG_EOGResources_vol_20d", "EQR_Equity_ret_1d", "EWG_Germany_ret_20d", "US7Y_Rate_ret_20d"], "is_new": true}, {"model_id": "new_h7_NORMAL_XGBoost_N25_t2", "algo": "XGBoost", "regime": "NORMAL", "horizon": 7, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWM_Malaysia_ret_1d", "EMR_Emerson_ret_20d", "Retail_Sales_zscore_60d", "SJM_JM_Smucker_ret_1d", "PLD_Prologis_ret_5d", "QQQ_vol_20d", "ASX_Australia_ret_5d", "LOW_Lowes_ret_20d", "MRK_Merck_zscore_60d", "AXP_Amex_vol_20d", "EWH_HongKong_ret_5d", "spx_abs_ret_max_5d", "CMCSA_ret_1d", "TM_Telephone_ret_1d", "Brent_Oil_FRED_ret_5d", "VVIX_ret_20d", "MSTR_Bitcoin3_ret_20d", "DHR_vol_20d", "HangSeng_HK_vol_20d", "hmm_p_stress", "Nikkei_Japan_vol_20d", "NEE_NextEra_ret_20d", "INTC_ret_1d"], "is_new": true}, {"model_id": "new_h7_NORMAL_XGBoost_N25_t3", "algo": "XGBoost", "regime": "NORMAL", "horizon": 7, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "GD_GeneralDynamics_zscore_60d", "VVIX_ret_20d", "hmm_p_stress", "SBUX_ret_5d", "LOW_Lowes_ret_20d", "ORCL_zscore_60d", "EWY_Korea_zscore_60d", "HD_zscore_60d", "IWM_SmallCap_vol_20d", "Brent_Oil_FRED_ret_5d", "BDX_Becton_Dickinson_ret_20d", "CPB_CampbellSoup_zscore_60d", "spx_abs_ret_max_5d", "US3M_Rate_zscore_60d", "US1Y_Rate_ret_5d", "heston_var_ev_h7", "JNJ_ret_1d", "LOW_Lowes_ret_5d", "LMT_LockheedMartin_ret_1d", "XLK_Tech_zscore_60d", "IYR_US_REIT2_zscore_60d", "NOC_Northrop_ret_20d", "DE_Deere_vol_20d"], "is_new": true}, {"model_id": "new_h7_NORMAL_XGBoost_N25_t4", "algo": "XGBoost", "regime": "NORMAL", "horizon": 7, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "US3M_Rate_zscore_60d", "CPB_CampbellSoup_ret_5d", "GILD_Gilead_ret_20d", "AMD_ret_5d", "CPB_CampbellSoup_vol_20d", "SPY_zscore_60d", "heston_var_ev_h7", "MSTR_Bitcoin3_ret_5d", "EXC_Exelon_ret_1d", "EWA_Australia_ret_1d", "LLY_zscore_60d", "PPL_PPL_ret_1d", "Brent_Oil_FRED_ret_5d", "NEE_NextEra_ret_20d", "XOM_ret_1d", "SJM_JM_Smucker_ret_1d", "spx_momentum_3d", "DIS_vol_20d", "WTI_Oil_FRED_zscore_60d", "XOM_ret_20d", "EQIX_Equinix_ret_5d", "XLK_Tech_zscore_60d", "DHR_ret_1d"], "is_new": true}, {"model_id": "new_h7_NORMAL_XGBoost_N25_t5", "algo": "XGBoost", "regime": "NORMAL", "horizon": 7, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "JNJ_ret_1d", "HD_ret_5d", "ORCL_vol_20d", "US5Y_Rate_ret_5d", "PPL_PPL_ret_1d", "EOG_EOGResources_vol_20d", "Nikkei_Japan_zscore_60d", "CLX_Clorox_vol_20d", "FedFunds_zscore_60d", "DHR_vol_20d", "GD_GeneralDynamics_zscore_60d", "HangSeng_HK_vol_20d", "T10Y2Y_Spread_ret_5d", "EWJ_Japan_vol_20d", "EXC_Exelon_ret_1d", "CPB_CampbellSoup_ret_5d", "PAYX_Paychex_zscore_60d", "LUV_SouthwestAir_ret_5d", "SCHW_Schwab_ret_5d", "EMR_Emerson_ret_20d", "EWL_Switzerland_vol_20d", "AXP_Amex_vol_20d", "US1Y_Rate_ret_5d"], "is_new": true}, {"model_id": "new_h7_NORMAL_XGBoost_N25_t6", "algo": "XGBoost", "regime": "NORMAL", "horizon": 7, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "heston_var_ev_h5", "NFCI_ret_5d", "VOD_Vodafone_zscore_60d", "HD_ret_20d", "GILD_Gilead_ret_20d", "MSTR_Bitcoin3_ret_5d", "LMT_LockheedMartin_ret_1d", "LOW_Lowes_ret_5d", "CMCSA_ret_1d", "PLD_Prologis_ret_5d", "EWL_Switzerland_vol_20d", "PPL_PPL_ret_1d", "DE_Deere_vol_20d", "EWY_Korea_ret_20d", "spx_momentum_3d", "XOM_ret_1d", "EWM_Malaysia_zscore_60d", "SLB_Schlumberger_ret_1d", "DIS_vol_20d", "EWG_Germany_ret_20d", "TXN_vol_20d", "EFFR_ret_1d", "IYR_US_REIT2_zscore_60d"], "is_new": true}, {"model_id": "new_h7_NORMAL_XGBoost_N25_t7", "algo": "XGBoost", "regime": "NORMAL", "horizon": 7, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "MSTR_Bitcoin3_ret_20d", "EWC_Canada_zscore_60d", "SBUX_ret_5d", "NEE_NextEra_ret_20d", "SJM_JM_Smucker_ret_5d", "EFFR_ret_1d", "TED_Spread_zscore_60d", "PPL_PPL_ret_1d", "GILD_Gilead_ret_20d", "hmm_p_stress", "INTC_ret_1d", "XLK_Tech_zscore_60d", "HD_ret_1d", "vix_acceleration_1d", "Retail_Sales_zscore_60d", "EWS_Singapore_ret_5d", "MO_AltriaMG_ret_1d", "US3M_Rate_zscore_60d", "Core_CPI_zscore_60d", "TGT_Target_zscore_60d", "BTI_BritishAmerican_ret_20d", "LMT_LockheedMartin_ret_1d", "SLB_Schlumberger_ret_1d"], "is_new": true}, {"model_id": "new_h7_NORMAL_XGBoost_N30_t0", "algo": "XGBoost", "regime": "NORMAL", "horizon": 7, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "Nikkei_Japan_vol_20d", "XOM_ret_20d", "MSTR_Bitcoin3_ret_5d", "vix_acceleration_1d", "EMR_Emerson_ret_20d", "SBUX_ret_5d", "EWM_Malaysia_ret_1d", "TXN_vol_20d", "PLD_Prologis_ret_5d", "DE_Deere_ret_5d", "FedFunds_zscore_60d", "T_ret_1d", "TM_Telephone_vol_20d", "NEE_NextEra_ret_20d", "EWA_Australia_ret_1d", "INTC_ret_5d", "VOD_Vodafone_zscore_60d", "DIS_vol_20d", "EQR_Equity_ret_1d", "CTAS_Cintas_vol_20d", "XLY_Disc_vol_20d", "US5Y_Rate_ret_5d", "EWM_Malaysia_vol_20d", "heston_var_ev_h7", "SCHW_Schwab_ret_5d", "LOW_Lowes_ret_5d", "XLV_Health_zscore_60d", "Michigan_Sentiment_ret_20d"], "is_new": true}, {"model_id": "new_h7_NORMAL_XGBoost_N30_t1", "algo": "XGBoost", "regime": "NORMAL", "horizon": 7, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "MS_MorganStanley_ret_5d", "WTI_Oil_FRED_zscore_60d", "XLK_Tech_zscore_60d", "MO_AltriaMG_ret_1d", "EWL_Switzerland_vol_20d", "PPL_PPL_ret_1d", "NWL_Newell_ret_20d", "SO_SouthernCo_ret_5d", "TM_Telephone_vol_20d", "EFFR_ret_1d", "HD_zscore_60d", "BA_ret_1d", "spx_vol_5d", "CLX_Clorox_vol_20d", "XLB_Materials_zscore_60d", "GD_GeneralDynamics_zscore_60d", "Core_CPI_zscore_60d", "EOG_EOGResources_ret_5d", "EMR_Emerson_ret_20d", "US7Y_Rate_ret_20d", "ITT_ITTInc_ret_5d", "HD_ret_5d", "PCAR_PaccarInc_ret_5d", "AMT_AmericanTower_ret_1d", "EWS_Singapore_ret_5d", "EXC_Exelon_ret_1d", "US1Y_Rate_ret_20d", "EWA_Australia_zscore_60d"], "is_new": true}, {"model_id": "new_h7_NORMAL_XGBoost_N30_t2", "algo": "XGBoost", "regime": "NORMAL", "horizon": 7, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "XLV_Health_zscore_60d", "XLB_Materials_zscore_60d", "ENB_EnbridgeInc_ret_1d", "US1Y_Rate_ret_20d", "US3Y_Rate_ret_5d", "SLB_Schlumberger_ret_1d", "EMR_Emerson_ret_20d", "CLX_Clorox_vol_20d", "PCAR_PaccarInc_ret_5d", "EWY_Korea_zscore_60d", "HD_ret_5d", "XLY_Disc_vol_20d", "FedFunds_zscore_60d", "LOW_Lowes_ret_20d", "XLK_Tech_zscore_60d", "SLB_Schlumberger_ret_5d", "MS_MorganStanley_zscore_60d", "SBUX_ret_5d", "EWL_Switzerland_vol_20d", "spx_abs_ret_max_5d", "DAX_Germany_zscore_60d", "CPB_CampbellSoup_ret_20d", "CPB_CampbellSoup_zscore_60d", "DHR_ret_1d", "CTAS_Cintas_vol_20d", "VVIX_ret_20d", "MS_MorganStanley_ret_5d", "BTI_BritishAmerican_ret_5d"], "is_new": true}, {"model_id": "new_h7_NORMAL_XGBoost_N30_t3", "algo": "XGBoost", "regime": "NORMAL", "horizon": 7, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "BLK_BlackRock_zscore_60d", "Brent_Oil_FRED_ret_20d", "AMD_ret_5d", "DHR_ret_1d", "WTI_Oil_FRED_zscore_60d", "ES_Evergy_ret_1d", "PLD_Prologis_ret_5d", "CTAS_Cintas_vol_20d", "TM_Telephone_ret_1d", "spx_momentum_3d", "DE_Deere_vol_20d", "XLY_Disc_vol_20d", "spx_vol_5d", "US6M_Rate_ret_20d", "HangSeng_HK_ret_5d", "FedFunds_zscore_60d", "MSTR_Bitcoin3_ret_20d", "SBUX_zscore_60d", "LLY_zscore_60d", "AMT_AmericanTower_ret_1d", "SBUX_vol_20d", "LUV_SouthwestAir_ret_5d", "AORD_AUS_zscore_60d", "GD_GeneralDynamics_zscore_60d", "Michigan_Sentiment_ret_20d", "vix_acceleration_1d", "gjr_condvar_h1", "DAX_Germany_zscore_60d"], "is_new": true}, {"model_id": "new_h7_NORMAL_XGBoost_N30_t4", "algo": "XGBoost", "regime": "NORMAL", "horizon": 7, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "CCI_CrownCastle_vol_20d", "Brent_Oil_FRED_ret_5d", "M_Macys_vol_20d", "SCHW_Schwab_ret_5d", "EWG_Germany_vol_20d", "XLK_Tech_zscore_60d", "gjr_condvar_h1", "Nikkei_Japan_vol_20d", "HangSeng_HK_vol_20d", "JNJ_ret_1d", "AXP_Amex_ret_20d", "MS_MorganStanley_zscore_60d", "SJM_JM_Smucker_ret_1d", "Nikkei_Japan_zscore_60d", "vix_mean_abs_ret_5d", "heston_var_ev_h3", "Michigan_Sentiment_ret_20d", "CTAS_Cintas_vol_20d", "CPB_CampbellSoup_ret_5d", "PCAR_PaccarInc_ret_5d", "PG_ret_20d", "MSTR_Bitcoin3_ret_20d", "US3Y_Rate_ret_5d", "MSTR_Bitcoin3_ret_1d", "EWH_HongKong_ret_5d", "XLV_Health_zscore_60d", "XLB_Materials_zscore_60d", "AORD_AUS_zscore_60d"], "is_new": true}, {"model_id": "new_h7_NORMAL_XGBoost_N30_t5", "algo": "XGBoost", "regime": "NORMAL", "horizon": 7, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "US3M_Rate_zscore_60d", "Industrial_Production_zscore_60d", "EFFR_ret_1d", "PFE_ret_1d", "EWL_Switzerland_vol_20d", "NEE_NextEra_ret_20d", "SBUX_ret_5d", "HangSeng_HK_ret_1d", "CPB_CampbellSoup_ret_20d", "EWA_Australia_ret_1d", "AXP_Amex_ret_20d", "GE_ret_1d", "LOW_Lowes_ret_20d", "3M_ret_5d", "EXC_Exelon_zscore_60d", "PCAR_PaccarInc_ret_5d", "EWY_Korea_zscore_60d", "GILD_Gilead_ret_20d", "DHR_ret_1d", "FedFunds_zscore_60d", "heston_ev_h3", "TED_Spread_vol_20d", "EWM_Malaysia_vol_20d", "PAYX_Paychex_ret_20d", "EWQ_France_zscore_60d", "XLY_Disc_vol_20d", "AMD_ret_5d", "SBUX_zscore_60d"], "is_new": true}, {"model_id": "new_h7_NORMAL_XGBoost_N30_t6", "algo": "XGBoost", "regime": "NORMAL", "horizon": 7, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "US6M_Rate_ret_20d", "Core_CPI_zscore_60d", "XOM_ret_20d", "BDX_Becton_Dickinson_ret_20d", "Nikkei_Japan_zscore_60d", "CPB_CampbellSoup_vol_20d", "TED_Spread_vol_20d", "EWG_Germany_vol_20d", "IYR_US_REIT2_zscore_60d", "LLY_zscore_60d", "EXC_Exelon_zscore_60d", "XLB_Materials_zscore_60d", "ORCL_vol_20d", "vix_mean_abs_ret_5d", "gjr_condvar_h1", "HD_ret_1d", "PLD_Prologis_ret_5d", "CTAS_Cintas_vol_20d", "3M_ret_5d", "PPL_PPL_ret_1d", "EFFR_vol_20d", "EWQ_France_zscore_60d", "LOW_Lowes_ret_20d", "heston_var_ev_h3", "ITT_ITTInc_ret_5d", "Core_PCE_zscore_60d", "BA_ret_1d", "PCAR_PaccarInc_ret_5d"], "is_new": true}, {"model_id": "new_h7_NORMAL_XGBoost_N30_t7", "algo": "XGBoost", "regime": "NORMAL", "horizon": 7, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "PCAR_PaccarInc_ret_5d", "3M_vol_20d", "ENB_EnbridgeInc_ret_1d", "SCHW_Schwab_ret_5d", "US30Y_Rate_ret_20d", "XLB_Materials_zscore_60d", "ES_Evergy_ret_1d", "VRP_ma5", "DIS_vol_20d", "CTAS_Cintas_vol_20d", "HangSeng_HK_vol_20d", "vix_mean_abs_ret_5d", "MSTR_Bitcoin3_ret_20d", "DE_Deere_ret_5d", "EFFR_vol_20d", "SPY_zscore_60d", "T10Y2Y_Spread_ret_5d", "US3M_Rate_zscore_60d", "VOD_Vodafone_zscore_60d", "heston_var_ev_h7", "HD_ret_1d", "EOG_EOGResources_vol_20d", "US1Y_Rate_ret_5d", "AMD_ret_5d", "EWC_Canada_zscore_60d", "CPB_CampbellSoup_ret_5d", "AXP_Amex_ret_20d", "SO_SouthernCo_ret_5d"], "is_new": true}, {"model_id": "new_h7_NORMAL_LightGBM_N5_t0", "algo": "LightGBM", "regime": "NORMAL", "horizon": 7, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "LMT_LockheedMartin_ret_1d", "US3Y_Rate_ret_5d", "EQIX_Equinix_ret_5d"], "is_new": true}, {"model_id": "new_h7_NORMAL_LightGBM_N5_t1", "algo": "LightGBM", "regime": "NORMAL", "horizon": 7, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "heston_var_ev_h3", "SLB_Schlumberger_ret_5d", "TGT_Target_zscore_60d"], "is_new": true}, {"model_id": "new_h7_NORMAL_LightGBM_N5_t2", "algo": "LightGBM", "regime": "NORMAL", "horizon": 7, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "SPY_zscore_60d", "US1Y_Rate_ret_5d", "VOD_Vodafone_zscore_60d"], "is_new": true}, {"model_id": "new_h7_NORMAL_LightGBM_N5_t3", "algo": "LightGBM", "regime": "NORMAL", "horizon": 7, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "ORCL_zscore_60d", "CTAS_Cintas_vol_20d", "SPY_zscore_60d"], "is_new": true}, {"model_id": "new_h7_NORMAL_LightGBM_N5_t4", "algo": "LightGBM", "regime": "NORMAL", "horizon": 7, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "NWL_Newell_ret_20d", "EWY_Korea_ret_20d", "LLY_zscore_60d"], "is_new": true}, {"model_id": "new_h7_NORMAL_LightGBM_N5_t5", "algo": "LightGBM", "regime": "NORMAL", "horizon": 7, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "heston_var_ev_h3", "Retail_Sales_zscore_60d", "VRP_ma5"], "is_new": true}, {"model_id": "new_h7_NORMAL_LightGBM_N5_t6", "algo": "LightGBM", "regime": "NORMAL", "horizon": 7, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "heston_var_ev_h7", "VRP_ma5", "EQR_Equity_ret_1d"], "is_new": true}, {"model_id": "new_h7_NORMAL_LightGBM_N5_t7", "algo": "LightGBM", "regime": "NORMAL", "horizon": 7, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "TED_Spread_zscore_60d", "MS_MorganStanley_ret_1d", "EOG_EOGResources_ret_5d"], "is_new": true}, {"model_id": "new_h7_NORMAL_LightGBM_N8_t0", "algo": "LightGBM", "regime": "NORMAL", "horizon": 7, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "CMCSA_ret_1d", "SLB_Schlumberger_ret_1d", "AXP_Amex_vol_20d", "EWA_Australia_zscore_60d", "TED_Spread_zscore_60d", "spx_vol_5d"], "is_new": true}, {"model_id": "new_h7_NORMAL_LightGBM_N8_t1", "algo": "LightGBM", "regime": "NORMAL", "horizon": 7, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "ASX_Australia_vol_20d", "MS_MorganStanley_ret_1d", "heston_ev_h3", "MSTR_Bitcoin3_ret_20d", "Michigan_Sentiment_ret_20d", "hmm_p_stress"], "is_new": true}, {"model_id": "new_h7_NORMAL_LightGBM_N8_t2", "algo": "LightGBM", "regime": "NORMAL", "horizon": 7, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "CPB_CampbellSoup_ret_5d", "US1Y_Rate_ret_20d", "EXC_Exelon_zscore_60d", "DE_Deere_ret_5d", "EWG_Germany_ret_20d", "TED_Spread_vol_20d"], "is_new": true}, {"model_id": "new_h7_NORMAL_LightGBM_N8_t3", "algo": "LightGBM", "regime": "NORMAL", "horizon": 7, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "AMD_ret_5d", "Michigan_Sentiment_ret_20d", "NFCI_ret_5d", "EWQ_France_zscore_60d", "DHR_ret_1d", "DE_Deere_ret_5d"], "is_new": true}, {"model_id": "new_h7_NORMAL_LightGBM_N8_t4", "algo": "LightGBM", "regime": "NORMAL", "horizon": 7, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "DAX_Germany_zscore_60d", "DOW_Price_zscore_60d", "EWH_HongKong_ret_5d", "Retail_Sales_zscore_60d", "SCHW_Schwab_ret_5d", "US7Y_Rate_ret_20d"], "is_new": true}, {"model_id": "new_h7_NORMAL_LightGBM_N8_t5", "algo": "LightGBM", "regime": "NORMAL", "horizon": 7, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "LOW_Lowes_ret_5d", "EXC_Exelon_ret_1d", "VOD_Vodafone_zscore_60d", "LMT_LockheedMartin_vol_20d", "ENB_EnbridgeInc_ret_1d", "EFFR_ret_1d"], "is_new": true}, {"model_id": "new_h7_NORMAL_LightGBM_N8_t6", "algo": "LightGBM", "regime": "NORMAL", "horizon": 7, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "XLY_Disc_vol_20d", "hmm_p_stress", "IBEX_Spain_ret_20d", "EFFR_vol_20d", "DHR_vol_20d", "AMD_ret_1d"], "is_new": true}, {"model_id": "new_h7_NORMAL_LightGBM_N8_t7", "algo": "LightGBM", "regime": "NORMAL", "horizon": 7, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "PPL_PPL_ret_1d", "LMT_LockheedMartin_ret_1d", "Core_PCE_zscore_60d", "EWM_Malaysia_zscore_60d", "3M_vol_20d", "heston_var_ev_h5"], "is_new": true}, {"model_id": "new_h7_NORMAL_LightGBM_N10_t0", "algo": "LightGBM", "regime": "NORMAL", "horizon": 7, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "DOW_Price_zscore_60d", "US3M_Rate_vol_20d", "PPL_PPL_ret_1d", "ENB_EnbridgeInc_ret_1d", "CLX_Clorox_vol_20d", "HangSeng_HK_ret_1d", "BLK_BlackRock_zscore_60d", "EQR_Equity_ret_1d"], "is_new": true}, {"model_id": "new_h7_NORMAL_LightGBM_N10_t1", "algo": "LightGBM", "regime": "NORMAL", "horizon": 7, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "TM_Telephone_vol_20d", "Core_PCE_zscore_60d", "HD_ret_5d", "MS_MorganStanley_zscore_60d", "EWY_Korea_ret_20d", "ITT_ITTInc_ret_5d", "IYM_BasicMaterials_ret_20d", "AMGN_Amgen_ret_1d"], "is_new": true}, {"model_id": "new_h7_NORMAL_LightGBM_N10_t2", "algo": "LightGBM", "regime": "NORMAL", "horizon": 7, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "HD_ret_20d", "AXP_Amex_ret_20d", "SBUX_ret_5d", "VOD_Vodafone_zscore_60d", "Core_CPI_zscore_60d", "EWQ_France_zscore_60d", "spx_momentum_3d", "PG_ret_20d"], "is_new": true}, {"model_id": "new_h7_NORMAL_LightGBM_N10_t3", "algo": "LightGBM", "regime": "NORMAL", "horizon": 7, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "Core_PCE_zscore_60d", "EWM_Malaysia_zscore_60d", "ENB_EnbridgeInc_ret_1d", "PAYX_Paychex_ret_20d", "hmm_p_stress", "INTC_ret_5d", "NWL_Newell_ret_20d", "NFCI_ret_5d"], "is_new": true}, {"model_id": "new_h7_NORMAL_LightGBM_N10_t4", "algo": "LightGBM", "regime": "NORMAL", "horizon": 7, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "ES_Evergy_ret_1d", "BTI_BritishAmerican_ret_5d", "MSTR_Bitcoin3_ret_20d", "TED_Spread_zscore_60d", "HD_ret_5d", "HD_ret_20d", "LMT_LockheedMartin_ret_1d", "MS_MorganStanley_ret_5d"], "is_new": true}, {"model_id": "new_h7_NORMAL_LightGBM_N10_t5", "algo": "LightGBM", "regime": "NORMAL", "horizon": 7, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "CPB_CampbellSoup_ret_5d", "EWC_Canada_zscore_60d", "EOG_EOGResources_ret_5d", "3M_ret_5d", "LUV_SouthwestAir_ret_5d", "SBUX_zscore_60d", "EWG_Germany_vol_20d", "PLD_Prologis_ret_5d"], "is_new": true}, {"model_id": "new_h7_NORMAL_LightGBM_N10_t6", "algo": "LightGBM", "regime": "NORMAL", "horizon": 7, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "ASX_Australia_ret_5d", "SBUX_zscore_60d", "CTAS_Cintas_vol_20d", "EQIX_Equinix_ret_5d", "SBUX_ret_5d", "PPL_PPL_ret_1d", "SJM_JM_Smucker_ret_1d", "Core_CPI_zscore_60d"], "is_new": true}, {"model_id": "new_h7_NORMAL_LightGBM_N10_t7", "algo": "LightGBM", "regime": "NORMAL", "horizon": 7, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "Nikkei_Japan_vol_20d", "EWY_Korea_zscore_60d", "hmm_p_stress", "MS_MorganStanley_ret_1d", "AVB_AvalonBay_zscore_60d", "TED_Spread_zscore_60d", "DHR_vol_20d", "spx_momentum_3d"], "is_new": true}, {"model_id": "new_h7_NORMAL_LightGBM_N12_t0", "algo": "LightGBM", "regime": "NORMAL", "horizon": 7, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "PAYX_Paychex_ret_20d", "DHR_ret_1d", "MS_MorganStanley_ret_5d", "LMT_LockheedMartin_vol_20d", "spx_vol_5d", "PFE_ret_1d", "IYR_US_REIT2_zscore_60d", "LUV_SouthwestAir_ret_5d", "EOG_EOGResources_vol_20d", "XLK_Tech_zscore_60d"], "is_new": true}, {"model_id": "new_h7_NORMAL_LightGBM_N12_t1", "algo": "LightGBM", "regime": "NORMAL", "horizon": 7, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "SCHW_Schwab_ret_5d", "EWQ_France_ret_20d", "EWG_Germany_vol_20d", "gjr_condvar_h1", "AVB_AvalonBay_zscore_60d", "EMR_Emerson_ret_20d", "EOG_EOGResources_vol_20d", "US3M_Rate_zscore_60d", "EWA_Australia_ret_1d", "3M_ret_5d"], "is_new": true}, {"model_id": "new_h7_NORMAL_LightGBM_N12_t2", "algo": "LightGBM", "regime": "NORMAL", "horizon": 7, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "CLX_Clorox_vol_20d", "CCI_CrownCastle_vol_20d", "Michigan_Sentiment_ret_20d", "AORD_AUS_zscore_60d", "SPY_zscore_60d", "MSTR_Bitcoin3_ret_1d", "heston_var_ev_h7", "DE_Deere_ret_5d", "Core_CPI_zscore_60d", "ORCL_vol_20d"], "is_new": true}, {"model_id": "new_h7_NORMAL_LightGBM_N12_t3", "algo": "LightGBM", "regime": "NORMAL", "horizon": 7, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "DHR_vol_20d", "PAYX_Paychex_ret_20d", "T10Y2Y_Spread_ret_5d", "Brent_Oil_FRED_ret_20d", "SLB_Schlumberger_ret_5d", "EWL_Switzerland_zscore_60d", "ES_Evergy_ret_1d", "IWM_SmallCap_vol_20d", "EOG_EOGResources_vol_20d", "IBEX_Spain_ret_20d"], "is_new": true}, {"model_id": "new_h7_NORMAL_LightGBM_N12_t4", "algo": "LightGBM", "regime": "NORMAL", "horizon": 7, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "SJM_JM_Smucker_ret_1d", "EQIX_Equinix_ret_5d", "Nikkei_Japan_vol_20d", "US1Y_Rate_ret_20d", "DE_Deere_ret_5d", "CCI_CrownCastle_vol_20d", "LMT_LockheedMartin_ret_1d", "heston_var_ev_h7", "IYR_US_REIT2_zscore_60d", "NEE_NextEra_ret_20d"], "is_new": true}, {"model_id": "new_h7_NORMAL_LightGBM_N12_t5", "algo": "LightGBM", "regime": "NORMAL", "horizon": 7, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "Brent_Oil_FRED_ret_5d", "EQR_Equity_ret_1d", "EXC_Exelon_zscore_60d", "SJM_JM_Smucker_ret_5d", "HangSeng_HK_vol_20d", "Core_CPI_zscore_60d", "PAYX_Paychex_ret_20d", "BTI_BritishAmerican_ret_20d", "hmm_p_stress", "LMT_LockheedMartin_vol_20d"], "is_new": true}, {"model_id": "new_h7_NORMAL_LightGBM_N12_t6", "algo": "LightGBM", "regime": "NORMAL", "horizon": 7, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "BTI_BritishAmerican_ret_20d", "TGT_Target_zscore_60d", "VRP_ma5", "AMZN_ret_5d", "SBUX_zscore_60d", "TXN_vol_20d", "EWA_Australia_ret_1d", "BTI_BritishAmerican_ret_5d", "INTC_ret_1d", "EFFR_ret_1d"], "is_new": true}, {"model_id": "new_h7_NORMAL_LightGBM_N12_t7", "algo": "LightGBM", "regime": "NORMAL", "horizon": 7, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "DHR_ret_1d", "NFCI_ret_5d", "VOD_Vodafone_zscore_60d", "WTI_Oil_FRED_zscore_60d", "MO_AltriaMG_ret_1d", "CPB_CampbellSoup_ret_5d", "US5Y_Rate_ret_5d", "spx_momentum_3d", "DAX_Germany_vol_20d", "EWA_Australia_zscore_60d"], "is_new": true}, {"model_id": "new_h7_NORMAL_LightGBM_N15_t0", "algo": "LightGBM", "regime": "NORMAL", "horizon": 7, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWJ_Japan_vol_20d", "SBUX_zscore_60d", "FedFunds_zscore_60d", "TED_Spread_zscore_60d", "LUV_SouthwestAir_ret_5d", "SO_SouthernCo_ret_5d", "AXP_Amex_vol_20d", "TED_Spread_vol_20d", "EQIX_Equinix_ret_5d", "XOM_ret_20d", "PG_ret_20d", "MS_MorganStanley_zscore_60d", "heston_var_ev_h3"], "is_new": true}, {"model_id": "new_h7_NORMAL_LightGBM_N15_t1", "algo": "LightGBM", "regime": "NORMAL", "horizon": 7, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "NEE_NextEra_ret_20d", "US6M_Rate_ret_20d", "EWG_Germany_vol_20d", "US3M_Rate_vol_20d", "3M_vol_20d", "WTI_Oil_FRED_zscore_60d", "CTAS_Cintas_vol_20d", "HangSeng_HK_ret_1d", "PLD_Prologis_ret_5d", "EQIX_Equinix_ret_5d", "AMZN_ret_5d", "AMD_ret_1d", "US30Y_Rate_ret_20d"], "is_new": true}, {"model_id": "new_h7_NORMAL_LightGBM_N15_t2", "algo": "LightGBM", "regime": "NORMAL", "horizon": 7, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "PCAR_PaccarInc_ret_5d", "EWM_Malaysia_vol_20d", "SJM_JM_Smucker_ret_1d", "heston_ev_h3", "TGT_Target_zscore_60d", "EOG_EOGResources_ret_5d", "Core_PCE_zscore_60d", "AXP_Amex_ret_20d", "IYM_BasicMaterials_ret_20d", "Michigan_Sentiment_ret_20d", "PFE_ret_1d", "EWM_Malaysia_zscore_60d", "EWJ_Japan_vol_20d"], "is_new": true}, {"model_id": "new_h7_NORMAL_LightGBM_N15_t3", "algo": "LightGBM", "regime": "NORMAL", "horizon": 7, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "IYM_BasicMaterials_ret_20d", "DAX_Germany_zscore_60d", "LOW_Lowes_ret_20d", "NFCI_ret_5d", "PPL_PPL_ret_1d", "AMGN_Amgen_ret_1d", "JNJ_ret_1d", "MRK_Merck_zscore_60d", "EWG_Germany_ret_20d", "HUM_Humana_ret_5d", "EWA_Australia_zscore_60d", "INTC_ret_1d", "EWS_Singapore_ret_5d"], "is_new": true}, {"model_id": "new_h7_NORMAL_LightGBM_N15_t4", "algo": "LightGBM", "regime": "NORMAL", "horizon": 7, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "DHR_ret_1d", "NWL_Newell_ret_20d", "SBUX_ret_5d", "CCI_CrownCastle_vol_20d", "INTC_ret_5d", "EWS_Singapore_ret_5d", "GILD_Gilead_ret_20d", "INTC_ret_1d", "IYR_US_REIT2_zscore_60d", "GE_ret_1d", "3M_vol_20d", "HUM_Humana_ret_5d", "EXC_Exelon_ret_1d"], "is_new": true}, {"model_id": "new_h7_NORMAL_LightGBM_N15_t5", "algo": "LightGBM", "regime": "NORMAL", "horizon": 7, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "BTI_BritishAmerican_ret_5d", "AXP_Amex_ret_20d", "spx_momentum_3d", "SO_SouthernCo_ret_5d", "EWC_Canada_zscore_60d", "SBUX_ret_5d", "US3M_Rate_vol_20d", "BLK_BlackRock_zscore_60d", "TED_Spread_zscore_60d", "PAYX_Paychex_zscore_60d", "US5Y_Rate_ret_5d", "US30Y_Rate_ret_20d", "ASX_Australia_vol_20d"], "is_new": true}, {"model_id": "new_h7_NORMAL_LightGBM_N15_t6", "algo": "LightGBM", "regime": "NORMAL", "horizon": 7, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "Nikkei_Japan_vol_20d", "PFE_ret_1d", "EWG_Germany_ret_20d", "T_ret_1d", "LUV_SouthwestAir_ret_5d", "Nikkei_Japan_zscore_60d", "HangSeng_HK_ret_1d", "vix_acceleration_1d", "T10Y2Y_Spread_ret_5d", "BDX_Becton_Dickinson_ret_20d", "AMZN_ret_5d", "Core_PCE_zscore_60d", "DHR_ret_1d"], "is_new": true}, {"model_id": "new_h7_NORMAL_LightGBM_N15_t7", "algo": "LightGBM", "regime": "NORMAL", "horizon": 7, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWL_Switzerland_vol_20d", "ES_Evergy_ret_1d", "CTAS_Cintas_vol_20d", "LOW_Lowes_ret_20d", "VVIX_ret_20d", "MSTR_Bitcoin3_ret_5d", "EWC_Canada_zscore_60d", "NOC_Northrop_ret_20d", "AORD_AUS_zscore_60d", "HD_zscore_60d", "3M_ret_5d", "DE_Deere_ret_5d", "HD_ret_5d"], "is_new": true}, {"model_id": "new_h7_NORMAL_LightGBM_N20_t0", "algo": "LightGBM", "regime": "NORMAL", "horizon": 7, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "VRP_ma5", "HangSeng_HK_ret_1d", "US3M_Rate_zscore_60d", "GILD_Gilead_ret_20d", "INTC_ret_5d", "DE_Deere_ret_5d", "SJM_JM_Smucker_ret_5d", "EWQ_France_ret_20d", "Brent_Oil_FRED_ret_5d", "IYM_BasicMaterials_ret_20d", "EWM_Malaysia_vol_20d", "EWQ_France_zscore_60d", "VVIX_ret_20d", "NFCI_ret_5d", "AMD_ret_1d", "CPB_CampbellSoup_vol_20d", "vix_mean_abs_ret_5d", "ASX_Australia_ret_5d"], "is_new": true}, {"model_id": "new_h7_NORMAL_LightGBM_N20_t1", "algo": "LightGBM", "regime": "NORMAL", "horizon": 7, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "ORCL_zscore_60d", "vix_acceleration_1d", "ITT_ITTInc_ret_5d", "EWG_Germany_ret_20d", "GILD_Gilead_ret_20d", "US3M_Rate_vol_20d", "IWM_SmallCap_vol_20d", "SPY_zscore_60d", "vix_mean_abs_ret_5d", "SBUX_ret_5d", "LOW_Lowes_ret_20d", "NOC_Northrop_ret_20d", "PLD_Prologis_ret_5d", "IBEX_Spain_ret_20d", "spx_vol_5d", "HangSeng_HK_vol_20d", "heston_var_ev_h3", "EWL_Switzerland_vol_20d"], "is_new": true}, {"model_id": "new_h7_NORMAL_LightGBM_N20_t2", "algo": "LightGBM", "regime": "NORMAL", "horizon": 7, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "ITT_ITTInc_ret_5d", "AMT_AmericanTower_ret_1d", "EWM_Malaysia_ret_1d", "DHR_vol_20d", "HD_zscore_60d", "MS_MorganStanley_zscore_60d", "PAYX_Paychex_zscore_60d", "Retail_Sales_zscore_60d", "SO_SouthernCo_ret_5d", "DAX_Germany_vol_20d", "SJM_JM_Smucker_ret_1d", "gjr_condvar_h1", "spx_momentum_3d", "DAX_Germany_zscore_60d", "MO_AltriaMG_ret_1d", "CCI_CrownCastle_vol_20d", "HUM_Humana_ret_5d", "SBUX_ret_5d"], "is_new": true}, {"model_id": "new_h7_NORMAL_LightGBM_N20_t3", "algo": "LightGBM", "regime": "NORMAL", "horizon": 7, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "PLD_Prologis_ret_5d", "SO_SouthernCo_ret_5d", "HangSeng_HK_vol_20d", "DOW_Price_zscore_60d", "MSTR_Bitcoin3_ret_20d", "INTC_ret_1d", "US6M_Rate_ret_20d", "PG_ret_20d", "XLB_Materials_zscore_60d", "SPY_zscore_60d", "IBEX_Spain_ret_20d", "EWM_Malaysia_vol_20d", "EQIX_Equinix_ret_5d", "ASX_Australia_ret_5d", "BTI_BritishAmerican_ret_5d", "Retail_Sales_zscore_60d", "CPB_CampbellSoup_ret_5d", "CTAS_Cintas_vol_20d"], "is_new": true}, {"model_id": "new_h7_NORMAL_LightGBM_N20_t4", "algo": "LightGBM", "regime": "NORMAL", "horizon": 7, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "BDX_Becton_Dickinson_ret_20d", "US30Y_Rate_ret_20d", "BTI_BritishAmerican_ret_5d", "US1Y_Rate_ret_5d", "EWM_Malaysia_vol_20d", "heston_var_ev_h7", "PAYX_Paychex_zscore_60d", "DIS_vol_20d", "MSTR_Bitcoin3_ret_20d", "BLK_BlackRock_zscore_60d", "spx_abs_ret_max_5d", "EWM_Malaysia_zscore_60d", "HangSeng_HK_ret_5d", "M_Macys_vol_20d", "EWC_Canada_zscore_60d", "DAX_Germany_vol_20d", "SO_SouthernCo_ret_5d", "XLV_Health_zscore_60d"], "is_new": true}, {"model_id": "new_h7_NORMAL_LightGBM_N20_t5", "algo": "LightGBM", "regime": "NORMAL", "horizon": 7, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "SBUX_zscore_60d", "EWS_Singapore_ret_5d", "gjr_condvar_h1", "VRP_ma5", "NWL_Newell_ret_20d", "EWL_Switzerland_zscore_60d", "ORCL_vol_20d", "US6M_Rate_ret_20d", "Nikkei_Japan_vol_20d", "XLF_Fin_vol_20d", "DAX_Germany_vol_20d", "ORCL_zscore_60d", "US1Y_Rate_ret_20d", "PAYX_Paychex_zscore_60d", "ASX_Australia_vol_20d", "AMGN_Amgen_ret_1d", "CPB_CampbellSoup_vol_20d", "PFE_ret_1d"], "is_new": true}, {"model_id": "new_h7_NORMAL_LightGBM_N20_t6", "algo": "LightGBM", "regime": "NORMAL", "horizon": 7, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "TM_Telephone_ret_1d", "TED_Spread_zscore_60d", "EWS_Singapore_ret_5d", "DE_Deere_ret_5d", "DOW_Price_zscore_60d", "CMCSA_ret_1d", "XOM_ret_1d", "QQQ_vol_20d", "IBEX_Spain_ret_20d", "3M_ret_5d", "EFFR_ret_1d", "SLB_Schlumberger_ret_5d", "DIS_vol_20d", "Brent_Oil_FRED_ret_5d", "Michigan_Sentiment_ret_20d", "GD_GeneralDynamics_zscore_60d", "EWY_Korea_ret_20d", "AORD_AUS_zscore_60d"], "is_new": true}, {"model_id": "new_h7_NORMAL_LightGBM_N20_t7", "algo": "LightGBM", "regime": "NORMAL", "horizon": 7, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "T_ret_1d", "Brent_Oil_FRED_ret_20d", "SO_SouthernCo_ret_5d", "AMGN_Amgen_ret_1d", "EFFR_vol_20d", "DAX_Germany_vol_20d", "gjr_condvar_h1", "AXP_Amex_vol_20d", "LMT_LockheedMartin_ret_1d", "BLK_BlackRock_zscore_60d", "EQR_Equity_ret_1d", "ITT_ITTInc_ret_5d", "EWY_Korea_ret_20d", "US1Y_Rate_ret_5d", "BTI_BritishAmerican_ret_5d", "ENB_EnbridgeInc_ret_1d", "AMD_ret_1d", "HangSeng_HK_vol_20d"], "is_new": true}, {"model_id": "new_h7_NORMAL_LightGBM_N25_t0", "algo": "LightGBM", "regime": "NORMAL", "horizon": 7, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "heston_ev_h3", "IYM_BasicMaterials_ret_20d", "XLY_Disc_vol_20d", "EWY_Korea_ret_20d", "MS_MorganStanley_zscore_60d", "spx_abs_ret_max_5d", "Industrial_Production_zscore_60d", "EXC_Exelon_zscore_60d", "LMT_LockheedMartin_vol_20d", "ITT_ITTInc_ret_5d", "ENB_EnbridgeInc_ret_1d", "BTI_BritishAmerican_ret_5d", "TED_Spread_zscore_60d", "AMD_ret_1d", "T_ret_1d", "INTC_ret_1d", "US6M_Rate_ret_20d", "heston_var_ev_h7", "HUM_Humana_ret_5d", "FedFunds_zscore_60d", "T10Y2Y_Spread_ret_5d", "ORCL_vol_20d", "EWQ_France_ret_20d"], "is_new": true}, {"model_id": "new_h7_NORMAL_LightGBM_N25_t1", "algo": "LightGBM", "regime": "NORMAL", "horizon": 7, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EQR_Equity_ret_1d", "VRP_ma5", "IYR_US_REIT2_zscore_60d", "ASX_Australia_ret_5d", "MS_MorganStanley_ret_5d", "AMZN_ret_5d", "TGT_Target_zscore_60d", "CPB_CampbellSoup_zscore_60d", "AXP_Amex_vol_20d", "XLV_Health_zscore_60d", "US7Y_Rate_ret_20d", "SJM_JM_Smucker_ret_5d", "IBEX_Spain_ret_20d", "IWM_SmallCap_vol_20d", "EQIX_Equinix_ret_5d", "Retail_Sales_zscore_60d", "NOC_Northrop_ret_20d", "CLX_Clorox_vol_20d", "TM_Telephone_vol_20d", "TM_Telephone_ret_1d", "AMD_ret_1d", "EOG_EOGResources_vol_20d", "FedFunds_zscore_60d"], "is_new": true}, {"model_id": "new_h7_NORMAL_LightGBM_N25_t2", "algo": "LightGBM", "regime": "NORMAL", "horizon": 7, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "GILD_Gilead_ret_20d", "CCI_CrownCastle_vol_20d", "TED_Spread_vol_20d", "FedFunds_zscore_60d", "DAX_Germany_zscore_60d", "GD_GeneralDynamics_zscore_60d", "EWL_Switzerland_vol_20d", "IWM_SmallCap_vol_20d", "AMGN_Amgen_ret_1d", "EXC_Exelon_zscore_60d", "SO_SouthernCo_ret_5d", "CTAS_Cintas_vol_20d", "HD_zscore_60d", "TXN_vol_20d", "LMT_LockheedMartin_ret_1d", "EWM_Malaysia_zscore_60d", "WTI_Oil_FRED_zscore_60d", "MS_MorganStanley_zscore_60d", "SLB_Schlumberger_ret_5d", "EWC_Canada_zscore_60d", "CI_Cigna_vol_20d", "T_ret_1d", "LUV_SouthwestAir_ret_5d"], "is_new": true}, {"model_id": "new_h7_NORMAL_LightGBM_N25_t3", "algo": "LightGBM", "regime": "NORMAL", "horizon": 7, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "HD_ret_1d", "PLD_Prologis_ret_5d", "EWM_Malaysia_ret_1d", "SBUX_vol_20d", "DE_Deere_ret_5d", "INTC_ret_5d", "PAYX_Paychex_zscore_60d", "PAYX_Paychex_ret_20d", "EWC_Canada_zscore_60d", "AMGN_Amgen_ret_1d", "ORCL_vol_20d", "EWM_Malaysia_zscore_60d", "US1Y_Rate_ret_20d", "MRK_Merck_zscore_60d", "LOW_Lowes_ret_5d", "MSTR_Bitcoin3_ret_20d", "BTI_BritishAmerican_ret_5d", "VOD_Vodafone_zscore_60d", "AVB_AvalonBay_zscore_60d", "TM_Telephone_vol_20d", "BDX_Becton_Dickinson_ret_20d", "MS_MorganStanley_ret_1d", "EWQ_France_ret_20d"], "is_new": true}, {"model_id": "new_h7_NORMAL_LightGBM_N25_t4", "algo": "LightGBM", "regime": "NORMAL", "horizon": 7, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "T_ret_1d", "QQQ_vol_20d", "XLB_Materials_zscore_60d", "PPL_PPL_ret_1d", "ORCL_zscore_60d", "EMR_Emerson_ret_20d", "MSTR_Bitcoin3_ret_1d", "EQIX_Equinix_ret_5d", "JNJ_ret_1d", "SBUX_zscore_60d", "NOC_Northrop_ret_20d", "EFFR_ret_1d", "DE_Deere_ret_5d", "US3M_Rate_zscore_60d", "SJM_JM_Smucker_ret_1d", "SJM_JM_Smucker_ret_5d", "hmm_p_stress", "HD_ret_1d", "SO_SouthernCo_ret_5d", "CMCSA_ret_1d", "vix_mean_abs_ret_5d", "DHR_ret_1d", "HangSeng_HK_ret_1d"], "is_new": true}, {"model_id": "new_h7_NORMAL_LightGBM_N25_t5", "algo": "LightGBM", "regime": "NORMAL", "horizon": 7, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EOG_EOGResources_ret_5d", "HD_ret_1d", "EFFR_ret_1d", "ASX_Australia_vol_20d", "BDX_Becton_Dickinson_ret_20d", "XLF_Fin_vol_20d", "ENB_EnbridgeInc_ret_1d", "WTI_Oil_FRED_zscore_60d", "heston_var_ev_h5", "NFCI_ret_5d", "gjr_condvar_h1", "hmm_p_stress", "LLY_zscore_60d", "spx_vol_5d", "TM_Telephone_ret_1d", "3M_vol_20d", "SJM_JM_Smucker_ret_1d", "spx_momentum_3d", "SBUX_ret_5d", "PFE_ret_1d", "BTI_BritishAmerican_ret_5d", "T10Y2Y_Spread_ret_5d", "TED_Spread_vol_20d"], "is_new": true}, {"model_id": "new_h7_NORMAL_LightGBM_N25_t6", "algo": "LightGBM", "regime": "NORMAL", "horizon": 7, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "DIS_vol_20d", "PFE_ret_1d", "VOD_Vodafone_zscore_60d", "EWG_Germany_vol_20d", "LOW_Lowes_ret_5d", "MS_MorganStanley_zscore_60d", "DHR_vol_20d", "IYR_US_REIT2_zscore_60d", "LOW_Lowes_ret_20d", "VRP_ma5", "CMCSA_ret_1d", "NVDA_vol_20d", "IYM_BasicMaterials_ret_20d", "IBEX_Spain_ret_20d", "EWA_Australia_zscore_60d", "FedFunds_zscore_60d", "NEE_NextEra_ret_20d", "PG_ret_20d", "BDX_Becton_Dickinson_ret_20d", "SBUX_zscore_60d", "EWM_Malaysia_vol_20d", "SLB_Schlumberger_ret_1d", "spx_momentum_3d"], "is_new": true}, {"model_id": "new_h7_NORMAL_LightGBM_N25_t7", "algo": "LightGBM", "regime": "NORMAL", "horizon": 7, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "SLB_Schlumberger_ret_5d", "CMCSA_ret_1d", "CPB_CampbellSoup_vol_20d", "AMGN_Amgen_ret_1d", "CTAS_Cintas_vol_20d", "US30Y_Rate_ret_20d", "ASX_Australia_ret_5d", "CLX_Clorox_vol_20d", "EWM_Malaysia_zscore_60d", "SO_SouthernCo_ret_5d", "AVB_AvalonBay_zscore_60d", "LOW_Lowes_ret_5d", "DHR_vol_20d", "AMD_ret_5d", "NOC_Northrop_ret_20d", "EWA_Australia_zscore_60d", "JNJ_ret_1d", "SCHW_Schwab_ret_5d", "DHR_ret_1d", "GE_ret_1d", "EWY_Korea_ret_20d", "HUM_Humana_ret_5d", "GD_GeneralDynamics_zscore_60d"], "is_new": true}, {"model_id": "new_h7_NORMAL_LightGBM_N30_t0", "algo": "LightGBM", "regime": "NORMAL", "horizon": 7, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "CLX_Clorox_vol_20d", "PAYX_Paychex_zscore_60d", "PAYX_Paychex_ret_20d", "EQIX_Equinix_ret_5d", "BLK_BlackRock_zscore_60d", "PFE_ret_1d", "HD_ret_20d", "HangSeng_HK_ret_1d", "SBUX_ret_5d", "DE_Deere_ret_5d", "DHR_ret_1d", "QQQ_vol_20d", "ORCL_vol_20d", "EXC_Exelon_zscore_60d", "EWY_Korea_ret_20d", "US3Y_Rate_ret_5d", "BTI_BritishAmerican_ret_20d", "CTAS_Cintas_vol_20d", "PPL_PPL_ret_1d", "EQR_Equity_ret_1d", "ENB_EnbridgeInc_ret_1d", "US30Y_Rate_ret_20d", "EWM_Malaysia_zscore_60d", "PAYX_Paychex_vol_20d", "IWM_SmallCap_vol_20d", "Core_PCE_zscore_60d", "TGT_Target_zscore_60d", "DE_Deere_vol_20d"], "is_new": true}, {"model_id": "new_h7_NORMAL_LightGBM_N30_t1", "algo": "LightGBM", "regime": "NORMAL", "horizon": 7, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "TED_Spread_zscore_60d", "IWM_SmallCap_vol_20d", "WTI_Oil_FRED_zscore_60d", "LLY_zscore_60d", "XLF_Fin_vol_20d", "EWC_Canada_zscore_60d", "TED_Spread_vol_20d", "US3M_Rate_vol_20d", "SJM_JM_Smucker_ret_1d", "US1Y_Rate_ret_20d", "Retail_Sales_zscore_60d", "DOW_Price_zscore_60d", "MSTR_Bitcoin3_ret_1d", "CPB_CampbellSoup_vol_20d", "spx_abs_ret_max_5d", "Nikkei_Japan_zscore_60d", "US6M_Rate_ret_20d", "SBUX_ret_5d", "Brent_Oil_FRED_ret_20d", "M_Macys_vol_20d", "vix_mean_abs_ret_5d", "PAYX_Paychex_vol_20d", "heston_var_ev_h3", "spx_momentum_3d", "FedFunds_zscore_60d", "Core_PCE_zscore_60d", "CI_Cigna_vol_20d", "SPY_zscore_60d"], "is_new": true}, {"model_id": "new_h7_NORMAL_LightGBM_N30_t2", "algo": "LightGBM", "regime": "NORMAL", "horizon": 7, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "VRP_ma5", "PAYX_Paychex_ret_20d", "spx_abs_ret_max_5d", "EXC_Exelon_zscore_60d", "US3Y_Rate_ret_5d", "EWY_Korea_zscore_60d", "XOM_ret_1d", "PAYX_Paychex_vol_20d", "XLK_Tech_zscore_60d", "CMCSA_ret_1d", "Michigan_Sentiment_ret_20d", "Core_CPI_zscore_60d", "SBUX_ret_5d", "CCI_CrownCastle_vol_20d", "3M_ret_5d", "EQIX_Equinix_ret_5d", "ORCL_zscore_60d", "Nikkei_Japan_zscore_60d", "EWS_Singapore_ret_5d", "AMD_ret_5d", "vix_mean_abs_ret_5d", "AXP_Amex_vol_20d", "CTAS_Cintas_vol_20d", "LOW_Lowes_ret_20d", "SBUX_vol_20d", "EWM_Malaysia_ret_1d", "AMD_ret_1d", "HUM_Humana_ret_5d"], "is_new": true}, {"model_id": "new_h7_NORMAL_LightGBM_N30_t3", "algo": "LightGBM", "regime": "NORMAL", "horizon": 7, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "VOD_Vodafone_zscore_60d", "CPB_CampbellSoup_ret_5d", "EWG_Germany_ret_20d", "EXC_Exelon_ret_1d", "EWH_HongKong_ret_5d", "SBUX_vol_20d", "INTC_ret_1d", "SBUX_ret_5d", "HD_zscore_60d", "NEE_NextEra_ret_20d", "SBUX_zscore_60d", "MSTR_Bitcoin3_ret_1d", "MSTR_Bitcoin3_ret_20d", "INTC_ret_5d", "HangSeng_HK_ret_1d", "DIS_vol_20d", "BLK_BlackRock_zscore_60d", "XLB_Materials_zscore_60d", "TGT_Target_zscore_60d", "M_Macys_vol_20d", "XOM_ret_20d", "MS_MorganStanley_ret_5d", "HD_ret_20d", "LLY_zscore_60d", "XLF_Fin_vol_20d", "ORCL_zscore_60d", "ASX_Australia_vol_20d", "SCHW_Schwab_ret_5d"], "is_new": true}, {"model_id": "new_h7_NORMAL_LightGBM_N30_t4", "algo": "LightGBM", "regime": "NORMAL", "horizon": 7, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "GD_GeneralDynamics_zscore_60d", "EMR_Emerson_ret_20d", "Brent_Oil_FRED_ret_20d", "AMZN_ret_5d", "EFFR_vol_20d", "3M_ret_5d", "VOD_Vodafone_zscore_60d", "ASX_Australia_vol_20d", "gjr_condvar_h1", "CMCSA_ret_1d", "MS_MorganStanley_zscore_60d", "QQQ_vol_20d", "HD_ret_5d", "HangSeng_HK_vol_20d", "EWH_HongKong_ret_5d", "MSTR_Bitcoin3_ret_20d", "CLX_Clorox_vol_20d", "EWM_Malaysia_ret_1d", "IWM_SmallCap_vol_20d", "EWC_Canada_zscore_60d", "T10Y2Y_Spread_ret_5d", "ITT_ITTInc_ret_5d", "SO_SouthernCo_ret_5d", "DAX_Germany_zscore_60d", "HUM_Humana_ret_5d", "AMD_ret_5d", "Nikkei_Japan_zscore_60d", "VRP_ma5"], "is_new": true}, {"model_id": "new_h7_NORMAL_LightGBM_N30_t5", "algo": "LightGBM", "regime": "NORMAL", "horizon": 7, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "IYR_US_REIT2_zscore_60d", "DHR_ret_1d", "EWY_Korea_zscore_60d", "M_Macys_vol_20d", "ES_Evergy_ret_1d", "GE_ret_1d", "ORCL_zscore_60d", "AVB_AvalonBay_zscore_60d", "spx_abs_ret_max_5d", "BTI_BritishAmerican_ret_5d", "SBUX_ret_5d", "PAYX_Paychex_vol_20d", "BDX_Becton_Dickinson_ret_20d", "DAX_Germany_zscore_60d", "IYM_BasicMaterials_ret_20d", "EWG_Germany_ret_20d", "TXN_vol_20d", "EWQ_France_zscore_60d", "Nikkei_Japan_vol_20d", "US6M_Rate_ret_20d", "SJM_JM_Smucker_ret_1d", "PAYX_Paychex_zscore_60d", "CPB_CampbellSoup_ret_5d", "AMGN_Amgen_ret_1d", "EQR_Equity_ret_1d", "DE_Deere_ret_5d", "AMD_ret_1d", "US30Y_Rate_ret_20d"], "is_new": true}, {"model_id": "new_h7_NORMAL_LightGBM_N30_t6", "algo": "LightGBM", "regime": "NORMAL", "horizon": 7, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "gjr_condvar_h1", "LUV_SouthwestAir_ret_5d", "BTI_BritishAmerican_ret_20d", "PPL_PPL_ret_1d", "HD_ret_20d", "XOM_ret_1d", "CPB_CampbellSoup_vol_20d", "GD_GeneralDynamics_zscore_60d", "MO_AltriaMG_ret_1d", "vix_acceleration_1d", "US3Y_Rate_ret_5d", "EOG_EOGResources_ret_5d", "LOW_Lowes_ret_20d", "EWA_Australia_ret_1d", "HangSeng_HK_ret_1d", "NOC_Northrop_ret_20d", "AMD_ret_1d", "IYR_US_REIT2_zscore_60d", "TM_Telephone_vol_20d", "CPB_CampbellSoup_ret_5d", "CPB_CampbellSoup_ret_20d", "INTC_ret_1d", "DHR_ret_1d", "EWL_Switzerland_zscore_60d", "NFCI_ret_5d", "CPB_CampbellSoup_zscore_60d", "EWQ_France_ret_20d", "SBUX_zscore_60d"], "is_new": true}, {"model_id": "new_h7_NORMAL_LightGBM_N30_t7", "algo": "LightGBM", "regime": "NORMAL", "horizon": 7, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EQR_Equity_ret_1d", "EXC_Exelon_ret_1d", "LMT_LockheedMartin_vol_20d", "XLV_Health_zscore_60d", "CI_Cigna_vol_20d", "DIS_vol_20d", "LOW_Lowes_ret_20d", "EWY_Korea_ret_20d", "MSTR_Bitcoin3_ret_20d", "TGT_Target_zscore_60d", "LLY_zscore_60d", "T_ret_1d", "NWL_Newell_ret_20d", "GD_GeneralDynamics_zscore_60d", "EWC_Canada_zscore_60d", "EWA_Australia_zscore_60d", "Core_CPI_zscore_60d", "Core_PCE_zscore_60d", "XLY_Disc_vol_20d", "MRK_Merck_zscore_60d", "EWL_Switzerland_vol_20d", "US5Y_Rate_ret_5d", "DHR_vol_20d", "BDX_Becton_Dickinson_ret_20d", "SCHW_Schwab_ret_5d", "Retail_Sales_zscore_60d", "BLK_BlackRock_zscore_60d", "TED_Spread_zscore_60d"], "is_new": true}, {"model_id": "new_h7_NORMAL_GradientBoosting_N5_t0", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 7, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "Nikkei_Japan_zscore_60d", "EWY_Korea_ret_20d", "ITT_ITTInc_ret_5d"], "is_new": true}, {"model_id": "new_h7_NORMAL_GradientBoosting_N5_t1", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 7, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "CMCSA_ret_1d", "heston_ev_h3", "EWY_Korea_zscore_60d"], "is_new": true}, {"model_id": "new_h7_NORMAL_GradientBoosting_N5_t2", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 7, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWC_Canada_zscore_60d", "TM_Telephone_vol_20d", "ORCL_vol_20d"], "is_new": true}, {"model_id": "new_h7_NORMAL_GradientBoosting_N5_t3", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 7, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "CLX_Clorox_vol_20d", "DHR_vol_20d", "TM_Telephone_ret_1d"], "is_new": true}, {"model_id": "new_h7_NORMAL_GradientBoosting_N5_t4", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 7, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "BDX_Becton_Dickinson_ret_20d", "SBUX_vol_20d", "MS_MorganStanley_ret_1d"], "is_new": true}, {"model_id": "new_h7_NORMAL_GradientBoosting_N5_t5", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 7, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "HD_zscore_60d", "US3Y_Rate_ret_5d", "AMD_ret_5d"], "is_new": true}, {"model_id": "new_h7_NORMAL_GradientBoosting_N5_t6", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 7, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWA_Australia_zscore_60d", "M_Macys_vol_20d", "DAX_Germany_vol_20d"], "is_new": true}, {"model_id": "new_h7_NORMAL_GradientBoosting_N5_t7", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 7, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "BLK_BlackRock_zscore_60d", "EOG_EOGResources_vol_20d", "LOW_Lowes_ret_5d"], "is_new": true}, {"model_id": "new_h7_NORMAL_GradientBoosting_N8_t0", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 7, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "T_ret_1d", "DOW_Price_zscore_60d", "HD_ret_20d", "BDX_Becton_Dickinson_ret_20d", "MSTR_Bitcoin3_ret_5d", "EWH_HongKong_ret_5d"], "is_new": true}, {"model_id": "new_h7_NORMAL_GradientBoosting_N8_t1", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 7, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "NWL_Newell_ret_20d", "BTI_BritishAmerican_ret_20d", "CPB_CampbellSoup_zscore_60d", "SBUX_zscore_60d", "HD_ret_5d", "XOM_ret_1d"], "is_new": true}, {"model_id": "new_h7_NORMAL_GradientBoosting_N8_t2", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 7, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "PG_ret_20d", "US3M_Rate_zscore_60d", "T10Y2Y_Spread_ret_5d", "VVIX_ret_20d", "PAYX_Paychex_zscore_60d", "gjr_condvar_h1"], "is_new": true}, {"model_id": "new_h7_NORMAL_GradientBoosting_N8_t3", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 7, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWM_Malaysia_zscore_60d", "CLX_Clorox_vol_20d", "ENB_EnbridgeInc_ret_1d", "INTC_ret_1d", "GILD_Gilead_ret_20d", "SBUX_zscore_60d"], "is_new": true}, {"model_id": "new_h7_NORMAL_GradientBoosting_N8_t4", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 7, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "US1Y_Rate_ret_20d", "AMT_AmericanTower_ret_1d", "PCAR_PaccarInc_ret_5d", "XLY_Disc_vol_20d", "EWM_Malaysia_ret_1d", "PPL_PPL_ret_1d"], "is_new": true}, {"model_id": "new_h7_NORMAL_GradientBoosting_N8_t5", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 7, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "DAX_Germany_vol_20d", "HangSeng_HK_vol_20d", "XOM_ret_20d", "Brent_Oil_FRED_ret_20d", "XLY_Disc_vol_20d", "TM_Telephone_ret_1d"], "is_new": true}, {"model_id": "new_h7_NORMAL_GradientBoosting_N8_t6", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 7, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "INTC_ret_1d", "AMGN_Amgen_ret_1d", "SCHW_Schwab_ret_5d", "TM_Telephone_ret_1d", "ITT_ITTInc_ret_5d", "EWA_Australia_ret_1d"], "is_new": true}, {"model_id": "new_h7_NORMAL_GradientBoosting_N8_t7", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 7, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "3M_vol_20d", "EQR_Equity_ret_1d", "AMD_ret_5d", "Nikkei_Japan_zscore_60d", "Retail_Sales_zscore_60d", "EFFR_ret_1d"], "is_new": true}, {"model_id": "new_h7_NORMAL_GradientBoosting_N10_t0", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 7, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EXC_Exelon_ret_1d", "NOC_Northrop_ret_20d", "VVIX_ret_20d", "MS_MorganStanley_ret_1d", "SBUX_zscore_60d", "MSTR_Bitcoin3_ret_1d", "PPL_PPL_ret_1d", "CCI_CrownCastle_vol_20d"], "is_new": true}, {"model_id": "new_h7_NORMAL_GradientBoosting_N10_t1", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 7, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "US1Y_Rate_ret_20d", "TXN_vol_20d", "QQQ_vol_20d", "T10Y2Y_Spread_ret_5d", "heston_var_ev_h5", "SLB_Schlumberger_ret_1d", "SBUX_ret_5d", "EWC_Canada_zscore_60d"], "is_new": true}, {"model_id": "new_h7_NORMAL_GradientBoosting_N10_t2", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 7, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "MSTR_Bitcoin3_ret_1d", "MO_AltriaMG_ret_1d", "SBUX_zscore_60d", "NOC_Northrop_ret_20d", "spx_vol_5d", "ASX_Australia_ret_5d", "heston_var_ev_h7", "DE_Deere_vol_20d"], "is_new": true}, {"model_id": "new_h7_NORMAL_GradientBoosting_N10_t3", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 7, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "NEE_NextEra_ret_20d", "US3M_Rate_vol_20d", "AMT_AmericanTower_ret_1d", "EMR_Emerson_ret_20d", "LUV_SouthwestAir_ret_5d", "WTI_Oil_FRED_zscore_60d", "spx_vol_5d", "MSTR_Bitcoin3_ret_20d"], "is_new": true}, {"model_id": "new_h7_NORMAL_GradientBoosting_N10_t4", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 7, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "HangSeng_HK_ret_5d", "AMD_ret_1d", "CCI_CrownCastle_vol_20d", "SBUX_vol_20d", "GD_GeneralDynamics_zscore_60d", "T_ret_1d", "VOD_Vodafone_zscore_60d", "BDX_Becton_Dickinson_ret_20d"], "is_new": true}, {"model_id": "new_h7_NORMAL_GradientBoosting_N10_t5", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 7, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "AMZN_ret_5d", "ES_Evergy_ret_1d", "US30Y_Rate_ret_20d", "SBUX_ret_5d", "SLB_Schlumberger_ret_5d", "EWM_Malaysia_ret_1d", "EWC_Canada_zscore_60d", "EMR_Emerson_ret_20d"], "is_new": true}, {"model_id": "new_h7_NORMAL_GradientBoosting_N10_t6", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 7, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "CLX_Clorox_vol_20d", "HangSeng_HK_vol_20d", "Nikkei_Japan_vol_20d", "Retail_Sales_zscore_60d", "XOM_ret_20d", "BLK_BlackRock_zscore_60d", "EWA_Australia_zscore_60d", "CMCSA_ret_1d"], "is_new": true}, {"model_id": "new_h7_NORMAL_GradientBoosting_N10_t7", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 7, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "DHR_vol_20d", "DAX_Germany_vol_20d", "HD_ret_1d", "SCHW_Schwab_ret_5d", "JNJ_ret_1d", "ENB_EnbridgeInc_ret_1d", "LLY_zscore_60d", "EQR_Equity_ret_1d"], "is_new": true}, {"model_id": "new_h7_NORMAL_GradientBoosting_N12_t0", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 7, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWA_Australia_zscore_60d", "ES_Evergy_ret_1d", "DHR_vol_20d", "ENB_EnbridgeInc_ret_1d", "Brent_Oil_FRED_ret_20d", "SBUX_ret_5d", "BDX_Becton_Dickinson_ret_20d", "vix_mean_abs_ret_5d", "LMT_LockheedMartin_ret_1d", "AXP_Amex_vol_20d"], "is_new": true}, {"model_id": "new_h7_NORMAL_GradientBoosting_N12_t1", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 7, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "GD_GeneralDynamics_zscore_60d", "SBUX_ret_5d", "US3M_Rate_vol_20d", "heston_var_ev_h5", "VRP_ma5", "GE_ret_1d", "BTI_BritishAmerican_ret_20d", "EFFR_vol_20d", "MSTR_Bitcoin3_ret_1d", "DAX_Germany_zscore_60d"], "is_new": true}, {"model_id": "new_h7_NORMAL_GradientBoosting_N12_t2", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 7, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "MO_AltriaMG_ret_1d", "EWQ_France_ret_20d", "DHR_vol_20d", "NVDA_vol_20d", "SO_SouthernCo_ret_5d", "MSTR_Bitcoin3_ret_5d", "NOC_Northrop_ret_20d", "PG_ret_20d", "VVIX_ret_20d", "BLK_BlackRock_zscore_60d"], "is_new": true}, {"model_id": "new_h7_NORMAL_GradientBoosting_N12_t3", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 7, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "TXN_vol_20d", "CPB_CampbellSoup_ret_5d", "GD_GeneralDynamics_zscore_60d", "PLD_Prologis_ret_5d", "MSTR_Bitcoin3_ret_20d", "DHR_ret_1d", "DE_Deere_vol_20d", "EOG_EOGResources_ret_5d", "CLX_Clorox_vol_20d", "EXC_Exelon_zscore_60d"], "is_new": true}, {"model_id": "new_h7_NORMAL_GradientBoosting_N12_t4", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 7, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "DE_Deere_ret_5d", "NEE_NextEra_ret_20d", "MS_MorganStanley_zscore_60d", "LMT_LockheedMartin_ret_1d", "EWA_Australia_ret_1d", "EWG_Germany_ret_20d", "AMZN_ret_5d", "MS_MorganStanley_ret_5d", "ORCL_zscore_60d", "PAYX_Paychex_zscore_60d"], "is_new": true}, {"model_id": "new_h7_NORMAL_GradientBoosting_N12_t5", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 7, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "QQQ_vol_20d", "IYR_US_REIT2_zscore_60d", "LMT_LockheedMartin_vol_20d", "NWL_Newell_ret_20d", "SPY_zscore_60d", "vix_mean_abs_ret_5d", "DHR_ret_1d", "IWM_SmallCap_vol_20d", "EXC_Exelon_zscore_60d", "EFFR_vol_20d"], "is_new": true}, {"model_id": "new_h7_NORMAL_GradientBoosting_N12_t6", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 7, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "hmm_p_stress", "INTC_ret_1d", "Brent_Oil_FRED_ret_5d", "Nikkei_Japan_zscore_60d", "TGT_Target_zscore_60d", "vix_mean_abs_ret_5d", "PLD_Prologis_ret_5d", "SBUX_vol_20d", "AVB_AvalonBay_zscore_60d", "AMGN_Amgen_ret_1d"], "is_new": true}, {"model_id": "new_h7_NORMAL_GradientBoosting_N12_t7", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 7, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "Nikkei_Japan_zscore_60d", "HUM_Humana_ret_5d", "XOM_ret_20d", "SPY_zscore_60d", "ASX_Australia_vol_20d", "EWM_Malaysia_ret_1d", "MO_AltriaMG_ret_1d", "Michigan_Sentiment_ret_20d", "ORCL_zscore_60d", "NWL_Newell_ret_20d"], "is_new": true}, {"model_id": "new_h7_NORMAL_GradientBoosting_N15_t0", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 7, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "HD_ret_20d", "BDX_Becton_Dickinson_ret_20d", "INTC_ret_5d", "TM_Telephone_ret_1d", "SCHW_Schwab_ret_5d", "EWJ_Japan_vol_20d", "IBEX_Spain_ret_20d", "EQIX_Equinix_ret_5d", "EMR_Emerson_ret_20d", "EQR_Equity_ret_1d", "LOW_Lowes_ret_20d", "EWA_Australia_ret_1d", "MS_MorganStanley_ret_5d"], "is_new": true}, {"model_id": "new_h7_NORMAL_GradientBoosting_N15_t1", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 7, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "DAX_Germany_vol_20d", "EWY_Korea_zscore_60d", "EWM_Malaysia_vol_20d", "BA_ret_1d", "XLY_Disc_vol_20d", "ASX_Australia_ret_5d", "EOG_EOGResources_vol_20d", "AXP_Amex_vol_20d", "M_Macys_vol_20d", "SLB_Schlumberger_ret_5d", "Brent_Oil_FRED_ret_5d", "US1Y_Rate_ret_20d", "EWL_Switzerland_vol_20d"], "is_new": true}, {"model_id": "new_h7_NORMAL_GradientBoosting_N15_t2", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 7, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "SPY_zscore_60d", "XOM_ret_1d", "PFE_ret_1d", "CPB_CampbellSoup_ret_5d", "vix_mean_abs_ret_5d", "EWS_Singapore_ret_5d", "SCHW_Schwab_ret_5d", "EWJ_Japan_vol_20d", "US30Y_Rate_ret_20d", "TXN_vol_20d", "SBUX_zscore_60d", "EFFR_ret_1d", "EWY_Korea_zscore_60d"], "is_new": true}, {"model_id": "new_h7_NORMAL_GradientBoosting_N15_t3", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 7, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "HangSeng_HK_ret_1d", "EWQ_France_zscore_60d", "EFFR_ret_1d", "EXC_Exelon_ret_1d", "3M_vol_20d", "JNJ_ret_1d", "ASX_Australia_vol_20d", "GD_GeneralDynamics_zscore_60d", "US5Y_Rate_ret_5d", "CLX_Clorox_vol_20d", "CPB_CampbellSoup_ret_5d", "EWY_Korea_ret_20d", "TGT_Target_zscore_60d"], "is_new": true}, {"model_id": "new_h7_NORMAL_GradientBoosting_N15_t4", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 7, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "QQQ_vol_20d", "US3Y_Rate_ret_5d", "PLD_Prologis_ret_5d", "heston_var_ev_h5", "DOW_Price_zscore_60d", "HD_ret_1d", "XOM_ret_20d", "BTI_BritishAmerican_ret_5d", "PAYX_Paychex_vol_20d", "EXC_Exelon_ret_1d", "EQR_Equity_ret_1d", "DE_Deere_ret_5d", "vix_acceleration_1d"], "is_new": true}, {"model_id": "new_h7_NORMAL_GradientBoosting_N15_t5", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 7, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "CTAS_Cintas_vol_20d", "BA_ret_1d", "CPB_CampbellSoup_zscore_60d", "3M_ret_5d", "DAX_Germany_zscore_60d", "BTI_BritishAmerican_ret_20d", "hmm_p_stress", "Core_CPI_zscore_60d", "EWL_Switzerland_zscore_60d", "vix_mean_abs_ret_5d", "EWG_Germany_ret_20d", "GD_GeneralDynamics_zscore_60d", "AVB_AvalonBay_zscore_60d"], "is_new": true}, {"model_id": "new_h7_NORMAL_GradientBoosting_N15_t6", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 7, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "Nikkei_Japan_vol_20d", "MS_MorganStanley_zscore_60d", "EWY_Korea_ret_20d", "EOG_EOGResources_vol_20d", "SBUX_zscore_60d", "INTC_ret_1d", "US3M_Rate_zscore_60d", "EWA_Australia_ret_1d", "FedFunds_zscore_60d", "SO_SouthernCo_ret_5d", "QQQ_vol_20d", "ITT_ITTInc_ret_5d", "LUV_SouthwestAir_ret_5d"], "is_new": true}, {"model_id": "new_h7_NORMAL_GradientBoosting_N15_t7", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 7, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "XLY_Disc_vol_20d", "SCHW_Schwab_ret_5d", "TM_Telephone_vol_20d", "PAYX_Paychex_vol_20d", "gjr_condvar_h1", "US6M_Rate_ret_20d", "US7Y_Rate_ret_20d", "ORCL_vol_20d", "EWS_Singapore_ret_5d", "EFFR_vol_20d", "EWQ_France_ret_20d", "T10Y2Y_Spread_ret_5d", "Nikkei_Japan_vol_20d"], "is_new": true}, {"model_id": "new_h7_NORMAL_GradientBoosting_N20_t0", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 7, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "gjr_condvar_h1", "EFFR_ret_1d", "HD_ret_1d", "IBEX_Spain_ret_20d", "CLX_Clorox_vol_20d", "PAYX_Paychex_ret_20d", "LMT_LockheedMartin_ret_1d", "EWH_HongKong_ret_5d", "VVIX_ret_20d", "NWL_Newell_ret_20d", "SBUX_ret_5d", "PLD_Prologis_ret_5d", "LOW_Lowes_ret_5d", "EWL_Switzerland_zscore_60d", "NFCI_ret_5d", "DHR_ret_1d", "Core_PCE_zscore_60d", "3M_vol_20d"], "is_new": true}, {"model_id": "new_h7_NORMAL_GradientBoosting_N20_t1", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 7, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "US1Y_Rate_ret_20d", "BTI_BritishAmerican_ret_20d", "BLK_BlackRock_zscore_60d", "ORCL_zscore_60d", "VRP_ma5", "DOW_Price_zscore_60d", "HUM_Humana_ret_5d", "SBUX_vol_20d", "MO_AltriaMG_ret_1d", "NEE_NextEra_ret_20d", "Retail_Sales_zscore_60d", "XOM_ret_20d", "3M_vol_20d", "PPL_PPL_ret_1d", "spx_vol_5d", "SBUX_ret_5d", "ASX_Australia_vol_20d", "ES_Evergy_ret_1d"], "is_new": true}, {"model_id": "new_h7_NORMAL_GradientBoosting_N20_t2", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 7, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "LOW_Lowes_ret_5d", "INTC_ret_1d", "ASX_Australia_ret_5d", "CCI_CrownCastle_vol_20d", "hmm_p_stress", "GE_ret_1d", "US1Y_Rate_ret_20d", "heston_ev_h3", "DHR_ret_1d", "EWG_Germany_vol_20d", "GILD_Gilead_ret_20d", "spx_momentum_3d", "PAYX_Paychex_vol_20d", "XLF_Fin_vol_20d", "US3M_Rate_vol_20d", "EWM_Malaysia_ret_1d", "DAX_Germany_vol_20d", "TGT_Target_zscore_60d"], "is_new": true}, {"model_id": "new_h7_NORMAL_GradientBoosting_N20_t3", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 7, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "HD_ret_20d", "EWA_Australia_ret_1d", "CLX_Clorox_vol_20d", "PAYX_Paychex_vol_20d", "BDX_Becton_Dickinson_ret_20d", "Nikkei_Japan_vol_20d", "SCHW_Schwab_ret_5d", "US1Y_Rate_ret_5d", "EWC_Canada_zscore_60d", "MSTR_Bitcoin3_ret_5d", "SBUX_vol_20d", "NWL_Newell_ret_20d", "AMGN_Amgen_ret_1d", "MS_MorganStanley_ret_5d", "US3Y_Rate_ret_5d", "T10Y2Y_Spread_ret_5d", "PCAR_PaccarInc_ret_5d", "US5Y_Rate_ret_5d"], "is_new": true}, {"model_id": "new_h7_NORMAL_GradientBoosting_N20_t4", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 7, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "BA_ret_1d", "TGT_Target_zscore_60d", "EOG_EOGResources_ret_5d", "ITT_ITTInc_ret_5d", "EWL_Switzerland_zscore_60d", "heston_var_ev_h7", "XLY_Disc_vol_20d", "ASX_Australia_vol_20d", "AMGN_Amgen_ret_1d", "LMT_LockheedMartin_ret_1d", "GD_GeneralDynamics_zscore_60d", "BTI_BritishAmerican_ret_20d", "MSTR_Bitcoin3_ret_1d", "hmm_p_stress", "PAYX_Paychex_ret_20d", "3M_vol_20d", "AXP_Amex_vol_20d", "FedFunds_zscore_60d"], "is_new": true}, {"model_id": "new_h7_NORMAL_GradientBoosting_N20_t5", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 7, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWY_Korea_zscore_60d", "AXP_Amex_vol_20d", "spx_momentum_3d", "ORCL_vol_20d", "NVDA_vol_20d", "AMT_AmericanTower_ret_1d", "Michigan_Sentiment_ret_20d", "AXP_Amex_ret_20d", "MSTR_Bitcoin3_ret_1d", "FedFunds_zscore_60d", "MSTR_Bitcoin3_ret_5d", "EMR_Emerson_ret_20d", "PLD_Prologis_ret_5d", "TED_Spread_vol_20d", "US1Y_Rate_ret_20d", "NWL_Newell_ret_20d", "NEE_NextEra_ret_20d", "EWG_Germany_ret_20d"], "is_new": true}, {"model_id": "new_h7_NORMAL_GradientBoosting_N20_t6", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 7, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWC_Canada_zscore_60d", "CPB_CampbellSoup_vol_20d", "EWA_Australia_zscore_60d", "PPL_PPL_ret_1d", "MO_AltriaMG_ret_1d", "TED_Spread_zscore_60d", "PFE_ret_1d", "SLB_Schlumberger_ret_1d", "AXP_Amex_vol_20d", "PG_ret_20d", "PAYX_Paychex_zscore_60d", "MS_MorganStanley_ret_1d", "PAYX_Paychex_ret_20d", "ORCL_zscore_60d", "vix_acceleration_1d", "LMT_LockheedMartin_vol_20d", "TM_Telephone_vol_20d", "EWQ_France_zscore_60d"], "is_new": true}, {"model_id": "new_h7_NORMAL_GradientBoosting_N20_t7", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 7, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "VOD_Vodafone_zscore_60d", "SBUX_zscore_60d", "SLB_Schlumberger_ret_5d", "LOW_Lowes_ret_5d", "IYR_US_REIT2_zscore_60d", "EFFR_ret_1d", "US5Y_Rate_ret_5d", "Nikkei_Japan_vol_20d", "FedFunds_zscore_60d", "TED_Spread_vol_20d", "vix_acceleration_1d", "XLV_Health_zscore_60d", "hmm_p_stress", "EWL_Switzerland_vol_20d", "T_ret_1d", "SLB_Schlumberger_ret_1d", "XOM_ret_20d", "NEE_NextEra_ret_20d"], "is_new": true}, {"model_id": "new_h7_NORMAL_GradientBoosting_N25_t0", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 7, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "ITT_ITTInc_ret_5d", "GE_ret_1d", "EWM_Malaysia_vol_20d", "HD_ret_5d", "BDX_Becton_Dickinson_ret_20d", "IWM_SmallCap_vol_20d", "GD_GeneralDynamics_zscore_60d", "CCI_CrownCastle_vol_20d", "M_Macys_vol_20d", "ES_Evergy_ret_1d", "hmm_p_stress", "LOW_Lowes_ret_5d", "US1Y_Rate_ret_20d", "spx_abs_ret_max_5d", "SO_SouthernCo_ret_5d", "Brent_Oil_FRED_ret_20d", "SCHW_Schwab_ret_5d", "SLB_Schlumberger_ret_1d", "heston_var_ev_h3", "VVIX_ret_20d", "SBUX_vol_20d", "Retail_Sales_zscore_60d", "DOW_Price_zscore_60d"], "is_new": true}, {"model_id": "new_h7_NORMAL_GradientBoosting_N25_t1", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 7, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "AORD_AUS_zscore_60d", "LUV_SouthwestAir_ret_5d", "CPB_CampbellSoup_vol_20d", "Brent_Oil_FRED_ret_5d", "3M_ret_5d", "EWG_Germany_ret_20d", "LMT_LockheedMartin_vol_20d", "EWJ_Japan_vol_20d", "LOW_Lowes_ret_20d", "US30Y_Rate_ret_20d", "EWQ_France_ret_20d", "XLV_Health_zscore_60d", "XOM_ret_20d", "IWM_SmallCap_vol_20d", "DE_Deere_vol_20d", "ITT_ITTInc_ret_5d", "SJM_JM_Smucker_ret_5d", "BLK_BlackRock_zscore_60d", "BA_ret_1d", "AMZN_ret_5d", "US3M_Rate_zscore_60d", "Nikkei_Japan_zscore_60d", "EWA_Australia_zscore_60d"], "is_new": true}, {"model_id": "new_h7_NORMAL_GradientBoosting_N25_t2", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 7, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWS_Singapore_ret_5d", "EWC_Canada_zscore_60d", "Brent_Oil_FRED_ret_5d", "Michigan_Sentiment_ret_20d", "MS_MorganStanley_ret_1d", "LLY_zscore_60d", "XLY_Disc_vol_20d", "EWM_Malaysia_ret_1d", "PAYX_Paychex_zscore_60d", "EWL_Switzerland_vol_20d", "INTC_ret_5d", "AXP_Amex_vol_20d", "CMCSA_ret_1d", "ASX_Australia_ret_5d", "EWH_HongKong_ret_5d", "SPY_zscore_60d", "AVB_AvalonBay_zscore_60d", "AMD_ret_1d", "M_Macys_vol_20d", "CPB_CampbellSoup_ret_20d", "IYR_US_REIT2_zscore_60d", "SJM_JM_Smucker_ret_5d", "QQQ_vol_20d"], "is_new": true}, {"model_id": "new_h7_NORMAL_GradientBoosting_N25_t3", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 7, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "HangSeng_HK_ret_1d", "spx_momentum_3d", "HD_ret_1d", "DAX_Germany_vol_20d", "US1Y_Rate_ret_5d", "XLY_Disc_vol_20d", "EFFR_vol_20d", "EWQ_France_zscore_60d", "spx_vol_5d", "TGT_Target_zscore_60d", "EOG_EOGResources_vol_20d", "DOW_Price_zscore_60d", "gjr_condvar_h1", "HUM_Humana_ret_5d", "PG_ret_20d", "CI_Cigna_vol_20d", "LMT_LockheedMartin_ret_1d", "IWM_SmallCap_vol_20d", "Core_PCE_zscore_60d", "INTC_ret_5d", "CMCSA_ret_1d", "Industrial_Production_zscore_60d", "heston_var_ev_h7"], "is_new": true}, {"model_id": "new_h7_NORMAL_GradientBoosting_N25_t4", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 7, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "XLV_Health_zscore_60d", "T10Y2Y_Spread_ret_5d", "ORCL_zscore_60d", "CMCSA_ret_1d", "CLX_Clorox_vol_20d", "XOM_ret_1d", "NEE_NextEra_ret_20d", "heston_ev_h3", "XLF_Fin_vol_20d", "AORD_AUS_zscore_60d", "DAX_Germany_zscore_60d", "Core_PCE_zscore_60d", "EWL_Switzerland_zscore_60d", "vix_acceleration_1d", "NWL_Newell_ret_20d", "TM_Telephone_vol_20d", "MS_MorganStanley_zscore_60d", "BTI_BritishAmerican_ret_5d", "CPB_CampbellSoup_vol_20d", "TXN_vol_20d", "EWM_Malaysia_zscore_60d", "US3M_Rate_vol_20d", "HangSeng_HK_vol_20d"], "is_new": true}, {"model_id": "new_h7_NORMAL_GradientBoosting_N25_t5", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 7, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "SBUX_zscore_60d", "CPB_CampbellSoup_ret_5d", "DHR_vol_20d", "US6M_Rate_ret_20d", "AMD_ret_1d", "HD_ret_1d", "EFFR_vol_20d", "HD_ret_5d", "AMGN_Amgen_ret_1d", "T10Y2Y_Spread_ret_5d", "EOG_EOGResources_vol_20d", "SJM_JM_Smucker_ret_1d", "HD_ret_20d", "MSTR_Bitcoin3_ret_20d", "CPB_CampbellSoup_vol_20d", "XLK_Tech_zscore_60d", "TED_Spread_zscore_60d", "DAX_Germany_vol_20d", "NWL_Newell_ret_20d", "IBEX_Spain_ret_20d", "HangSeng_HK_ret_1d", "EWM_Malaysia_zscore_60d", "EWG_Germany_vol_20d"], "is_new": true}, {"model_id": "new_h7_NORMAL_GradientBoosting_N25_t6", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 7, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "XLF_Fin_vol_20d", "AXP_Amex_vol_20d", "ASX_Australia_ret_5d", "VRP_ma5", "spx_vol_5d", "M_Macys_vol_20d", "EOG_EOGResources_vol_20d", "XLY_Disc_vol_20d", "WTI_Oil_FRED_zscore_60d", "CI_Cigna_vol_20d", "EWM_Malaysia_ret_1d", "vix_mean_abs_ret_5d", "vix_acceleration_1d", "EXC_Exelon_zscore_60d", "CPB_CampbellSoup_ret_5d", "AMD_ret_5d", "HD_ret_1d", "BA_ret_1d", "MS_MorganStanley_ret_1d", "EWM_Malaysia_zscore_60d", "EWL_Switzerland_vol_20d", "CTAS_Cintas_vol_20d", "XOM_ret_20d"], "is_new": true}, {"model_id": "new_h7_NORMAL_GradientBoosting_N25_t7", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 7, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "AMT_AmericanTower_ret_1d", "AXP_Amex_vol_20d", "XLK_Tech_zscore_60d", "CPB_CampbellSoup_vol_20d", "Michigan_Sentiment_ret_20d", "HD_zscore_60d", "MSTR_Bitcoin3_ret_20d", "Brent_Oil_FRED_ret_5d", "XLV_Health_zscore_60d", "heston_var_ev_h5", "INTC_ret_1d", "US30Y_Rate_ret_20d", "LOW_Lowes_ret_20d", "EWJ_Japan_vol_20d", "ASX_Australia_vol_20d", "SPY_zscore_60d", "NVDA_vol_20d", "PLD_Prologis_ret_5d", "EWY_Korea_zscore_60d", "MS_MorganStanley_ret_5d", "EWS_Singapore_ret_5d", "heston_ev_h3", "US5Y_Rate_ret_5d"], "is_new": true}, {"model_id": "new_h7_NORMAL_GradientBoosting_N30_t0", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 7, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "Nikkei_Japan_vol_20d", "DAX_Germany_zscore_60d", "CPB_CampbellSoup_vol_20d", "XLF_Fin_vol_20d", "AMD_ret_1d", "EWA_Australia_ret_1d", "NEE_NextEra_ret_20d", "Brent_Oil_FRED_ret_5d", "MS_MorganStanley_ret_1d", "IYR_US_REIT2_zscore_60d", "GD_GeneralDynamics_zscore_60d", "EWM_Malaysia_ret_1d", "SLB_Schlumberger_ret_5d", "HUM_Humana_ret_5d", "EWY_Korea_zscore_60d", "Core_PCE_zscore_60d", "DE_Deere_ret_5d", "ORCL_vol_20d", "XOM_ret_20d", "EMR_Emerson_ret_20d", "NVDA_vol_20d", "EQR_Equity_ret_1d", "EWM_Malaysia_zscore_60d", "Retail_Sales_zscore_60d", "NOC_Northrop_ret_20d", "EWG_Germany_vol_20d", "gjr_condvar_h1", "IWM_SmallCap_vol_20d"], "is_new": true}, {"model_id": "new_h7_NORMAL_GradientBoosting_N30_t1", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 7, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWG_Germany_vol_20d", "MO_AltriaMG_ret_1d", "XOM_ret_20d", "EWH_HongKong_ret_5d", "IYR_US_REIT2_zscore_60d", "DAX_Germany_zscore_60d", "ENB_EnbridgeInc_ret_1d", "PFE_ret_1d", "heston_var_ev_h5", "FedFunds_zscore_60d", "TED_Spread_vol_20d", "ASX_Australia_vol_20d", "SBUX_vol_20d", "NWL_Newell_ret_20d", "MS_MorganStanley_zscore_60d", "NFCI_ret_5d", "AXP_Amex_ret_20d", "US3M_Rate_vol_20d", "NEE_NextEra_ret_20d", "INTC_ret_5d", "HangSeng_HK_vol_20d", "LMT_LockheedMartin_vol_20d", "US3Y_Rate_ret_5d", "heston_var_ev_h7", "VVIX_ret_20d", "GD_GeneralDynamics_zscore_60d", "HD_ret_20d", "Industrial_Production_zscore_60d"], "is_new": true}, {"model_id": "new_h7_NORMAL_GradientBoosting_N30_t2", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 7, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "HangSeng_HK_ret_1d", "ENB_EnbridgeInc_ret_1d", "vix_mean_abs_ret_5d", "VOD_Vodafone_zscore_60d", "EWQ_France_zscore_60d", "EQR_Equity_ret_1d", "Retail_Sales_zscore_60d", "spx_abs_ret_max_5d", "US1Y_Rate_ret_20d", "EXC_Exelon_ret_1d", "TGT_Target_zscore_60d", "TXN_vol_20d", "QQQ_vol_20d", "NVDA_vol_20d", "Brent_Oil_FRED_ret_5d", "3M_vol_20d", "BDX_Becton_Dickinson_ret_20d", "INTC_ret_1d", "EWY_Korea_zscore_60d", "TM_Telephone_ret_1d", "DHR_vol_20d", "XOM_ret_1d", "AVB_AvalonBay_zscore_60d", "XLY_Disc_vol_20d", "ASX_Australia_vol_20d", "heston_var_ev_h3", "PFE_ret_1d", "CTAS_Cintas_vol_20d"], "is_new": true}, {"model_id": "new_h7_NORMAL_GradientBoosting_N30_t3", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 7, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "SLB_Schlumberger_ret_5d", "LLY_zscore_60d", "SO_SouthernCo_ret_5d", "BA_ret_1d", "DOW_Price_zscore_60d", "3M_ret_5d", "T10Y2Y_Spread_ret_5d", "EFFR_vol_20d", "AXP_Amex_vol_20d", "Michigan_Sentiment_ret_20d", "HD_zscore_60d", "MS_MorganStanley_ret_1d", "AMT_AmericanTower_ret_1d", "DE_Deere_ret_5d", "TED_Spread_vol_20d", "SPY_zscore_60d", "EWC_Canada_zscore_60d", "NEE_NextEra_ret_20d", "XLK_Tech_zscore_60d", "LOW_Lowes_ret_5d", "SBUX_vol_20d", "TGT_Target_zscore_60d", "NFCI_ret_5d", "ASX_Australia_vol_20d", "CLX_Clorox_vol_20d", "vix_acceleration_1d", "heston_var_ev_h7", "AMD_ret_1d"], "is_new": true}, {"model_id": "new_h7_NORMAL_GradientBoosting_N30_t4", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 7, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "spx_abs_ret_max_5d", "SLB_Schlumberger_ret_5d", "Nikkei_Japan_zscore_60d", "US6M_Rate_ret_20d", "NOC_Northrop_ret_20d", "PPL_PPL_ret_1d", "IYR_US_REIT2_zscore_60d", "3M_vol_20d", "HangSeng_HK_ret_1d", "heston_var_ev_h3", "NVDA_vol_20d", "GE_ret_1d", "EWH_HongKong_ret_5d", "BA_ret_1d", "SCHW_Schwab_ret_5d", "SJM_JM_Smucker_ret_1d", "EQR_Equity_ret_1d", "AXP_Amex_vol_20d", "PLD_Prologis_ret_5d", "SO_SouthernCo_ret_5d", "XLB_Materials_zscore_60d", "ES_Evergy_ret_1d", "TXN_vol_20d", "EFFR_vol_20d", "PFE_ret_1d", "EWM_Malaysia_ret_1d", "US3M_Rate_zscore_60d", "TGT_Target_zscore_60d"], "is_new": true}, {"model_id": "new_h7_NORMAL_GradientBoosting_N30_t5", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 7, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "PFE_ret_1d", "IBEX_Spain_ret_20d", "PAYX_Paychex_vol_20d", "MS_MorganStanley_ret_1d", "XOM_ret_1d", "EWH_HongKong_ret_5d", "AMZN_ret_5d", "HD_ret_5d", "Industrial_Production_zscore_60d", "Brent_Oil_FRED_ret_20d", "ASX_Australia_vol_20d", "LOW_Lowes_ret_5d", "TM_Telephone_ret_1d", "EWL_Switzerland_vol_20d", "SBUX_vol_20d", "T_ret_1d", "vix_acceleration_1d", "WTI_Oil_FRED_zscore_60d", "SO_SouthernCo_ret_5d", "Nikkei_Japan_vol_20d", "AVB_AvalonBay_zscore_60d", "BLK_BlackRock_zscore_60d", "MRK_Merck_zscore_60d", "HangSeng_HK_ret_1d", "DHR_ret_1d", "XOM_ret_20d", "heston_ev_h3", "TED_Spread_zscore_60d"], "is_new": true}, {"model_id": "new_h7_NORMAL_GradientBoosting_N30_t6", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 7, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "LMT_LockheedMartin_ret_1d", "AMGN_Amgen_ret_1d", "SBUX_zscore_60d", "PPL_PPL_ret_1d", "SLB_Schlumberger_ret_5d", "TM_Telephone_vol_20d", "FedFunds_zscore_60d", "EWM_Malaysia_vol_20d", "US6M_Rate_ret_20d", "US3Y_Rate_ret_5d", "MO_AltriaMG_ret_1d", "AMT_AmericanTower_ret_1d", "XLF_Fin_vol_20d", "CPB_CampbellSoup_ret_5d", "TGT_Target_zscore_60d", "CCI_CrownCastle_vol_20d", "HD_zscore_60d", "TED_Spread_zscore_60d", "EWJ_Japan_vol_20d", "IYM_BasicMaterials_ret_20d", "AXP_Amex_ret_20d", "EOG_EOGResources_ret_5d", "NWL_Newell_ret_20d", "LMT_LockheedMartin_vol_20d", "EFFR_ret_1d", "XLB_Materials_zscore_60d", "Retail_Sales_zscore_60d", "ASX_Australia_vol_20d"], "is_new": true}, {"model_id": "new_h7_NORMAL_GradientBoosting_N30_t7", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 7, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "vix_mean_abs_ret_5d", "PAYX_Paychex_vol_20d", "PAYX_Paychex_ret_20d", "SBUX_vol_20d", "US30Y_Rate_ret_20d", "CTAS_Cintas_vol_20d", "AXP_Amex_vol_20d", "TGT_Target_zscore_60d", "T_ret_1d", "CPB_CampbellSoup_zscore_60d", "EWS_Singapore_ret_5d", "AVB_AvalonBay_zscore_60d", "MRK_Merck_zscore_60d", "ORCL_zscore_60d", "JNJ_ret_1d", "GE_ret_1d", "CPB_CampbellSoup_ret_5d", "AMT_AmericanTower_ret_1d", "AMD_ret_1d", "EWG_Germany_ret_20d", "XLF_Fin_vol_20d", "NFCI_ret_5d", "US7Y_Rate_ret_20d", "MS_MorganStanley_ret_1d", "EFFR_ret_1d", "Brent_Oil_FRED_ret_5d", "EWG_Germany_vol_20d", "EQIX_Equinix_ret_5d"], "is_new": true}, {"model_id": "new_h7_NORMAL_RandomForest_N5_t0", "algo": "RandomForest", "regime": "NORMAL", "horizon": 7, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "CI_Cigna_vol_20d", "EXC_Exelon_zscore_60d", "DE_Deere_ret_5d"], "is_new": true}, {"model_id": "new_h7_NORMAL_RandomForest_N5_t1", "algo": "RandomForest", "regime": "NORMAL", "horizon": 7, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "XLB_Materials_zscore_60d", "GILD_Gilead_ret_20d", "EFFR_vol_20d"], "is_new": true}, {"model_id": "new_h7_NORMAL_RandomForest_N5_t2", "algo": "RandomForest", "regime": "NORMAL", "horizon": 7, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "NWL_Newell_ret_20d", "HUM_Humana_ret_5d", "IYM_BasicMaterials_ret_20d"], "is_new": true}, {"model_id": "new_h7_NORMAL_RandomForest_N5_t3", "algo": "RandomForest", "regime": "NORMAL", "horizon": 7, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EMR_Emerson_ret_20d", "HangSeng_HK_ret_1d", "PAYX_Paychex_zscore_60d"], "is_new": true}, {"model_id": "new_h7_NORMAL_RandomForest_N5_t4", "algo": "RandomForest", "regime": "NORMAL", "horizon": 7, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWG_Germany_ret_20d", "IYR_US_REIT2_zscore_60d", "PAYX_Paychex_vol_20d"], "is_new": true}, {"model_id": "new_h7_NORMAL_RandomForest_N5_t5", "algo": "RandomForest", "regime": "NORMAL", "horizon": 7, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWQ_France_ret_20d", "DHR_ret_1d", "EWG_Germany_ret_20d"], "is_new": true}, {"model_id": "new_h7_NORMAL_RandomForest_N5_t6", "algo": "RandomForest", "regime": "NORMAL", "horizon": 7, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "CPB_CampbellSoup_ret_5d", "IYM_BasicMaterials_ret_20d", "GD_GeneralDynamics_zscore_60d"], "is_new": true}, {"model_id": "new_h7_NORMAL_RandomForest_N5_t7", "algo": "RandomForest", "regime": "NORMAL", "horizon": 7, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "TED_Spread_vol_20d", "PAYX_Paychex_ret_20d", "IBEX_Spain_ret_20d"], "is_new": true}, {"model_id": "new_h7_NORMAL_RandomForest_N8_t0", "algo": "RandomForest", "regime": "NORMAL", "horizon": 7, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "XLY_Disc_vol_20d", "AMD_ret_1d", "MSTR_Bitcoin3_ret_20d", "CPB_CampbellSoup_zscore_60d", "GD_GeneralDynamics_zscore_60d", "EWL_Switzerland_vol_20d"], "is_new": true}, {"model_id": "new_h7_NORMAL_RandomForest_N8_t1", "algo": "RandomForest", "regime": "NORMAL", "horizon": 7, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "AMZN_ret_5d", "TM_Telephone_ret_1d", "EFFR_ret_1d", "EWQ_France_ret_20d", "EQR_Equity_ret_1d", "heston_var_ev_h3"], "is_new": true}, {"model_id": "new_h7_NORMAL_RandomForest_N8_t2", "algo": "RandomForest", "regime": "NORMAL", "horizon": 7, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "AMD_ret_5d", "SPY_zscore_60d", "ES_Evergy_ret_1d", "HUM_Humana_ret_5d", "heston_var_ev_h7", "SLB_Schlumberger_ret_5d"], "is_new": true}, {"model_id": "new_h7_NORMAL_RandomForest_N8_t3", "algo": "RandomForest", "regime": "NORMAL", "horizon": 7, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "CCI_CrownCastle_vol_20d", "NEE_NextEra_ret_20d", "EWL_Switzerland_vol_20d", "LOW_Lowes_ret_20d", "IYR_US_REIT2_zscore_60d", "LLY_zscore_60d"], "is_new": true}, {"model_id": "new_h7_NORMAL_RandomForest_N8_t4", "algo": "RandomForest", "regime": "NORMAL", "horizon": 7, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "LLY_zscore_60d", "PCAR_PaccarInc_ret_5d", "HD_ret_5d", "Retail_Sales_zscore_60d", "EWQ_France_zscore_60d", "US30Y_Rate_ret_20d"], "is_new": true}, {"model_id": "new_h7_NORMAL_RandomForest_N8_t5", "algo": "RandomForest", "regime": "NORMAL", "horizon": 7, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "Retail_Sales_zscore_60d", "Brent_Oil_FRED_ret_5d", "MSTR_Bitcoin3_ret_5d", "EWM_Malaysia_ret_1d", "HangSeng_HK_ret_5d", "GD_GeneralDynamics_zscore_60d"], "is_new": true}, {"model_id": "new_h7_NORMAL_RandomForest_N8_t6", "algo": "RandomForest", "regime": "NORMAL", "horizon": 7, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWG_Germany_ret_20d", "CI_Cigna_vol_20d", "US3Y_Rate_ret_5d", "ORCL_zscore_60d", "LLY_zscore_60d", "PAYX_Paychex_vol_20d"], "is_new": true}, {"model_id": "new_h7_NORMAL_RandomForest_N8_t7", "algo": "RandomForest", "regime": "NORMAL", "horizon": 7, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "IWM_SmallCap_vol_20d", "EWG_Germany_ret_20d", "PLD_Prologis_ret_5d", "BA_ret_1d", "SJM_JM_Smucker_ret_1d", "GE_ret_1d"], "is_new": true}, {"model_id": "new_h7_NORMAL_RandomForest_N10_t0", "algo": "RandomForest", "regime": "NORMAL", "horizon": 7, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "Brent_Oil_FRED_ret_5d", "EWY_Korea_zscore_60d", "EWL_Switzerland_vol_20d", "US6M_Rate_ret_20d", "NFCI_ret_5d", "US3M_Rate_zscore_60d", "DOW_Price_zscore_60d", "CI_Cigna_vol_20d"], "is_new": true}, {"model_id": "new_h7_NORMAL_RandomForest_N10_t1", "algo": "RandomForest", "regime": "NORMAL", "horizon": 7, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "heston_ev_h3", "XLB_Materials_zscore_60d", "EWM_Malaysia_zscore_60d", "ASX_Australia_vol_20d", "PAYX_Paychex_ret_20d", "vix_acceleration_1d", "MO_AltriaMG_ret_1d", "AXP_Amex_ret_20d"], "is_new": true}, {"model_id": "new_h7_NORMAL_RandomForest_N10_t2", "algo": "RandomForest", "regime": "NORMAL", "horizon": 7, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "BLK_BlackRock_zscore_60d", "US3M_Rate_vol_20d", "heston_var_ev_h3", "ORCL_zscore_60d", "HangSeng_HK_vol_20d", "DAX_Germany_vol_20d", "FedFunds_zscore_60d", "SCHW_Schwab_ret_5d"], "is_new": true}, {"model_id": "new_h7_NORMAL_RandomForest_N10_t3", "algo": "RandomForest", "regime": "NORMAL", "horizon": 7, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "heston_var_ev_h7", "spx_vol_5d", "CPB_CampbellSoup_ret_5d", "IYR_US_REIT2_zscore_60d", "WTI_Oil_FRED_zscore_60d", "LOW_Lowes_ret_20d", "XLK_Tech_zscore_60d", "EWY_Korea_ret_20d"], "is_new": true}, {"model_id": "new_h7_NORMAL_RandomForest_N10_t4", "algo": "RandomForest", "regime": "NORMAL", "horizon": 7, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "VVIX_ret_20d", "PAYX_Paychex_ret_20d", "INTC_ret_5d", "CLX_Clorox_vol_20d", "M_Macys_vol_20d", "GD_GeneralDynamics_zscore_60d", "EWY_Korea_zscore_60d", "PAYX_Paychex_vol_20d"], "is_new": true}, {"model_id": "new_h7_NORMAL_RandomForest_N10_t5", "algo": "RandomForest", "regime": "NORMAL", "horizon": 7, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWC_Canada_zscore_60d", "XLK_Tech_zscore_60d", "PG_ret_20d", "MSTR_Bitcoin3_ret_1d", "EFFR_vol_20d", "CI_Cigna_vol_20d", "AMT_AmericanTower_ret_1d", "SO_SouthernCo_ret_5d"], "is_new": true}, {"model_id": "new_h7_NORMAL_RandomForest_N10_t6", "algo": "RandomForest", "regime": "NORMAL", "horizon": 7, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "IBEX_Spain_ret_20d", "ENB_EnbridgeInc_ret_1d", "AXP_Amex_ret_20d", "SJM_JM_Smucker_ret_5d", "HangSeng_HK_ret_1d", "US30Y_Rate_ret_20d", "Core_PCE_zscore_60d", "EWM_Malaysia_zscore_60d"], "is_new": true}, {"model_id": "new_h7_NORMAL_RandomForest_N10_t7", "algo": "RandomForest", "regime": "NORMAL", "horizon": 7, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "HD_ret_1d", "SBUX_zscore_60d", "PAYX_Paychex_ret_20d", "DAX_Germany_vol_20d", "SLB_Schlumberger_ret_5d", "Nikkei_Japan_zscore_60d", "QQQ_vol_20d", "EWM_Malaysia_vol_20d"], "is_new": true}, {"model_id": "new_h7_NORMAL_RandomForest_N12_t0", "algo": "RandomForest", "regime": "NORMAL", "horizon": 7, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "MSTR_Bitcoin3_ret_1d", "EQIX_Equinix_ret_5d", "EWM_Malaysia_ret_1d", "XLV_Health_zscore_60d", "heston_var_ev_h3", "EQR_Equity_ret_1d", "GD_GeneralDynamics_zscore_60d", "CLX_Clorox_vol_20d", "LOW_Lowes_ret_20d", "EWQ_France_zscore_60d"], "is_new": true}, {"model_id": "new_h7_NORMAL_RandomForest_N12_t1", "algo": "RandomForest", "regime": "NORMAL", "horizon": 7, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "GILD_Gilead_ret_20d", "XLF_Fin_vol_20d", "vix_acceleration_1d", "PG_ret_20d", "CPB_CampbellSoup_vol_20d", "EWG_Germany_vol_20d", "heston_var_ev_h3", "Nikkei_Japan_zscore_60d", "EXC_Exelon_zscore_60d", "Brent_Oil_FRED_ret_5d"], "is_new": true}, {"model_id": "new_h7_NORMAL_RandomForest_N12_t2", "algo": "RandomForest", "regime": "NORMAL", "horizon": 7, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "T10Y2Y_Spread_ret_5d", "VOD_Vodafone_zscore_60d", "JNJ_ret_1d", "AMGN_Amgen_ret_1d", "CTAS_Cintas_vol_20d", "M_Macys_vol_20d", "GE_ret_1d", "US1Y_Rate_ret_20d", "Retail_Sales_zscore_60d", "QQQ_vol_20d"], "is_new": true}, {"model_id": "new_h7_NORMAL_RandomForest_N12_t3", "algo": "RandomForest", "regime": "NORMAL", "horizon": 7, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "CPB_CampbellSoup_vol_20d", "PAYX_Paychex_zscore_60d", "US5Y_Rate_ret_5d", "EWH_HongKong_ret_5d", "US30Y_Rate_ret_20d", "heston_var_ev_h7", "LMT_LockheedMartin_vol_20d", "DHR_ret_1d", "LOW_Lowes_ret_5d", "DHR_vol_20d"], "is_new": true}, {"model_id": "new_h7_NORMAL_RandomForest_N12_t4", "algo": "RandomForest", "regime": "NORMAL", "horizon": 7, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "AMD_ret_1d", "HD_ret_1d", "MSTR_Bitcoin3_ret_20d", "IYM_BasicMaterials_ret_20d", "PAYX_Paychex_zscore_60d", "CMCSA_ret_1d", "heston_ev_h3", "PFE_ret_1d", "EWM_Malaysia_ret_1d", "heston_var_ev_h3"], "is_new": true}, {"model_id": "new_h7_NORMAL_RandomForest_N12_t5", "algo": "RandomForest", "regime": "NORMAL", "horizon": 7, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "PCAR_PaccarInc_ret_5d", "TGT_Target_zscore_60d", "heston_var_ev_h5", "PAYX_Paychex_ret_20d", "EQIX_Equinix_ret_5d", "US3M_Rate_zscore_60d", "EWM_Malaysia_zscore_60d", "3M_ret_5d", "MS_MorganStanley_ret_5d", "3M_vol_20d"], "is_new": true}, {"model_id": "new_h7_NORMAL_RandomForest_N12_t6", "algo": "RandomForest", "regime": "NORMAL", "horizon": 7, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "VVIX_ret_20d", "EWA_Australia_ret_1d", "SLB_Schlumberger_ret_1d", "EOG_EOGResources_ret_5d", "US6M_Rate_ret_20d", "NWL_Newell_ret_20d", "EWH_HongKong_ret_5d", "MRK_Merck_zscore_60d", "INTC_ret_1d", "EWM_Malaysia_zscore_60d"], "is_new": true}, {"model_id": "new_h7_NORMAL_RandomForest_N12_t7", "algo": "RandomForest", "regime": "NORMAL", "horizon": 7, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "AMD_ret_5d", "DHR_ret_1d", "DE_Deere_vol_20d", "BTI_BritishAmerican_ret_5d", "DHR_vol_20d", "heston_ev_h3", "FedFunds_zscore_60d", "EXC_Exelon_ret_1d", "NFCI_ret_5d", "IBEX_Spain_ret_20d"], "is_new": true}, {"model_id": "new_h7_NORMAL_RandomForest_N15_t0", "algo": "RandomForest", "regime": "NORMAL", "horizon": 7, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "MSTR_Bitcoin3_ret_20d", "XLY_Disc_vol_20d", "spx_abs_ret_max_5d", "spx_vol_5d", "GD_GeneralDynamics_zscore_60d", "Industrial_Production_zscore_60d", "US3M_Rate_zscore_60d", "IBEX_Spain_ret_20d", "SLB_Schlumberger_ret_1d", "EWH_HongKong_ret_5d", "AORD_AUS_zscore_60d", "Retail_Sales_zscore_60d", "GILD_Gilead_ret_20d"], "is_new": true}, {"model_id": "new_h7_NORMAL_RandomForest_N15_t1", "algo": "RandomForest", "regime": "NORMAL", "horizon": 7, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "LMT_LockheedMartin_vol_20d", "EXC_Exelon_zscore_60d", "DE_Deere_ret_5d", "DIS_vol_20d", "SLB_Schlumberger_ret_5d", "US7Y_Rate_ret_20d", "PPL_PPL_ret_1d", "FedFunds_zscore_60d", "EWH_HongKong_ret_5d", "EXC_Exelon_ret_1d", "SBUX_ret_5d", "M_Macys_vol_20d", "EFFR_ret_1d"], "is_new": true}, {"model_id": "new_h7_NORMAL_RandomForest_N15_t2", "algo": "RandomForest", "regime": "NORMAL", "horizon": 7, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "vix_acceleration_1d", "ORCL_zscore_60d", "Core_PCE_zscore_60d", "EWQ_France_ret_20d", "XLV_Health_zscore_60d", "MS_MorganStanley_ret_5d", "AXP_Amex_ret_20d", "US1Y_Rate_ret_20d", "EWA_Australia_zscore_60d", "SPY_zscore_60d", "VRP_ma5", "DAX_Germany_zscore_60d", "BTI_BritishAmerican_ret_5d"], "is_new": true}, {"model_id": "new_h7_NORMAL_RandomForest_N15_t3", "algo": "RandomForest", "regime": "NORMAL", "horizon": 7, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "HD_ret_1d", "Brent_Oil_FRED_ret_20d", "MSTR_Bitcoin3_ret_20d", "MRK_Merck_zscore_60d", "Michigan_Sentiment_ret_20d", "QQQ_vol_20d", "EWC_Canada_zscore_60d", "SLB_Schlumberger_ret_5d", "spx_vol_5d", "Nikkei_Japan_vol_20d", "US1Y_Rate_ret_20d", "AMD_ret_5d", "IWM_SmallCap_vol_20d"], "is_new": true}, {"model_id": "new_h7_NORMAL_RandomForest_N15_t4", "algo": "RandomForest", "regime": "NORMAL", "horizon": 7, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "ENB_EnbridgeInc_ret_1d", "DAX_Germany_vol_20d", "PFE_ret_1d", "IYM_BasicMaterials_ret_20d", "NVDA_vol_20d", "ES_Evergy_ret_1d", "Nikkei_Japan_vol_20d", "WTI_Oil_FRED_zscore_60d", "DIS_vol_20d", "EWG_Germany_vol_20d", "EWM_Malaysia_ret_1d", "US30Y_Rate_ret_20d", "EWL_Switzerland_zscore_60d"], "is_new": true}, {"model_id": "new_h7_NORMAL_RandomForest_N15_t5", "algo": "RandomForest", "regime": "NORMAL", "horizon": 7, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "HangSeng_HK_vol_20d", "NVDA_vol_20d", "SBUX_ret_5d", "MSTR_Bitcoin3_ret_1d", "SJM_JM_Smucker_ret_1d", "EWA_Australia_zscore_60d", "ORCL_zscore_60d", "US5Y_Rate_ret_5d", "TM_Telephone_ret_1d", "US1Y_Rate_ret_20d", "VRP_ma5", "AMT_AmericanTower_ret_1d", "3M_ret_5d"], "is_new": true}, {"model_id": "new_h7_NORMAL_RandomForest_N15_t6", "algo": "RandomForest", "regime": "NORMAL", "horizon": 7, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "US30Y_Rate_ret_20d", "MRK_Merck_zscore_60d", "Core_PCE_zscore_60d", "MO_AltriaMG_ret_1d", "XLF_Fin_vol_20d", "MSTR_Bitcoin3_ret_1d", "VVIX_ret_20d", "US1Y_Rate_ret_20d", "CPB_CampbellSoup_vol_20d", "EWH_HongKong_ret_5d", "DE_Deere_ret_5d", "ENB_EnbridgeInc_ret_1d", "vix_mean_abs_ret_5d"], "is_new": true}, {"model_id": "new_h7_NORMAL_RandomForest_N15_t7", "algo": "RandomForest", "regime": "NORMAL", "horizon": 7, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "GILD_Gilead_ret_20d", "EWA_Australia_zscore_60d", "ENB_EnbridgeInc_ret_1d", "US7Y_Rate_ret_20d", "US3M_Rate_zscore_60d", "AMZN_ret_5d", "EWA_Australia_ret_1d", "EFFR_ret_1d", "IYR_US_REIT2_zscore_60d", "EWJ_Japan_vol_20d", "NVDA_vol_20d", "CPB_CampbellSoup_zscore_60d", "IBEX_Spain_ret_20d"], "is_new": true}, {"model_id": "new_h7_NORMAL_RandomForest_N20_t0", "algo": "RandomForest", "regime": "NORMAL", "horizon": 7, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "Brent_Oil_FRED_ret_20d", "ITT_ITTInc_ret_5d", "SLB_Schlumberger_ret_1d", "VRP_ma5", "IYM_BasicMaterials_ret_20d", "vix_mean_abs_ret_5d", "EQR_Equity_ret_1d", "PAYX_Paychex_vol_20d", "SBUX_ret_5d", "SPY_zscore_60d", "EWL_Switzerland_zscore_60d", "HD_zscore_60d", "BLK_BlackRock_zscore_60d", "HD_ret_5d", "NEE_NextEra_ret_20d", "NVDA_vol_20d", "LMT_LockheedMartin_vol_20d", "BDX_Becton_Dickinson_ret_20d"], "is_new": true}, {"model_id": "new_h7_NORMAL_RandomForest_N20_t1", "algo": "RandomForest", "regime": "NORMAL", "horizon": 7, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "TED_Spread_zscore_60d", "EQIX_Equinix_ret_5d", "EWM_Malaysia_vol_20d", "EOG_EOGResources_ret_5d", "NFCI_ret_5d", "NOC_Northrop_ret_20d", "TM_Telephone_ret_1d", "ASX_Australia_vol_20d", "T_ret_1d", "MSTR_Bitcoin3_ret_1d", "MS_MorganStanley_ret_5d", "EWG_Germany_ret_20d", "CI_Cigna_vol_20d", "LOW_Lowes_ret_20d", "EFFR_vol_20d", "heston_var_ev_h3", "SBUX_vol_20d", "CPB_CampbellSoup_ret_5d"], "is_new": true}, {"model_id": "new_h7_NORMAL_RandomForest_N20_t2", "algo": "RandomForest", "regime": "NORMAL", "horizon": 7, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "HD_ret_5d", "Nikkei_Japan_zscore_60d", "MS_MorganStanley_ret_1d", "EOG_EOGResources_ret_5d", "ASX_Australia_vol_20d", "EWA_Australia_zscore_60d", "Industrial_Production_zscore_60d", "NFCI_ret_5d", "MSTR_Bitcoin3_ret_20d", "BLK_BlackRock_zscore_60d", "US3M_Rate_vol_20d", "FedFunds_zscore_60d", "GE_ret_1d", "XLK_Tech_zscore_60d", "SBUX_zscore_60d", "TXN_vol_20d", "US6M_Rate_ret_20d", "MO_AltriaMG_ret_1d"], "is_new": true}, {"model_id": "new_h7_NORMAL_RandomForest_N20_t3", "algo": "RandomForest", "regime": "NORMAL", "horizon": 7, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "QQQ_vol_20d", "XOM_ret_1d", "NFCI_ret_5d", "DOW_Price_zscore_60d", "NEE_NextEra_ret_20d", "DE_Deere_vol_20d", "LUV_SouthwestAir_ret_5d", "PAYX_Paychex_vol_20d", "TED_Spread_zscore_60d", "XLV_Health_zscore_60d", "SBUX_vol_20d", "Core_PCE_zscore_60d", "heston_var_ev_h3", "MSTR_Bitcoin3_ret_1d", "heston_var_ev_h7", "US1Y_Rate_ret_5d", "DIS_vol_20d", "US3M_Rate_zscore_60d"], "is_new": true}, {"model_id": "new_h7_NORMAL_RandomForest_N20_t4", "algo": "RandomForest", "regime": "NORMAL", "horizon": 7, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "3M_ret_5d", "SLB_Schlumberger_ret_1d", "PAYX_Paychex_zscore_60d", "CPB_CampbellSoup_zscore_60d", "SBUX_vol_20d", "gjr_condvar_h1", "MS_MorganStanley_zscore_60d", "spx_momentum_3d", "TXN_vol_20d", "EFFR_ret_1d", "Core_CPI_zscore_60d", "LMT_LockheedMartin_ret_1d", "ITT_ITTInc_ret_5d", "SLB_Schlumberger_ret_5d", "T_ret_1d", "MSTR_Bitcoin3_ret_1d", "AMT_AmericanTower_ret_1d", "EFFR_vol_20d"], "is_new": true}, {"model_id": "new_h7_NORMAL_RandomForest_N20_t5", "algo": "RandomForest", "regime": "NORMAL", "horizon": 7, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "QQQ_vol_20d", "CPB_CampbellSoup_zscore_60d", "GILD_Gilead_ret_20d", "spx_momentum_3d", "Core_CPI_zscore_60d", "CCI_CrownCastle_vol_20d", "EWM_Malaysia_zscore_60d", "MS_MorganStanley_zscore_60d", "EWS_Singapore_ret_5d", "SCHW_Schwab_ret_5d", "PAYX_Paychex_ret_20d", "Nikkei_Japan_zscore_60d", "EQR_Equity_ret_1d", "MS_MorganStanley_ret_1d", "3M_vol_20d", "CPB_CampbellSoup_ret_20d", "T10Y2Y_Spread_ret_5d", "EWM_Malaysia_vol_20d"], "is_new": true}, {"model_id": "new_h7_NORMAL_RandomForest_N20_t6", "algo": "RandomForest", "regime": "NORMAL", "horizon": 7, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "PFE_ret_1d", "AMGN_Amgen_ret_1d", "TM_Telephone_ret_1d", "SBUX_ret_5d", "BLK_BlackRock_zscore_60d", "BTI_BritishAmerican_ret_5d", "EXC_Exelon_zscore_60d", "LUV_SouthwestAir_ret_5d", "AMD_ret_1d", "NEE_NextEra_ret_20d", "LOW_Lowes_ret_5d", "US1Y_Rate_ret_5d", "3M_vol_20d", "HD_ret_20d", "MS_MorganStanley_zscore_60d", "US3M_Rate_vol_20d", "INTC_ret_5d", "QQQ_vol_20d"], "is_new": true}, {"model_id": "new_h7_NORMAL_RandomForest_N20_t7", "algo": "RandomForest", "regime": "NORMAL", "horizon": 7, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "MSTR_Bitcoin3_ret_20d", "ORCL_vol_20d", "MS_MorganStanley_zscore_60d", "EXC_Exelon_ret_1d", "BTI_BritishAmerican_ret_5d", "IWM_SmallCap_vol_20d", "LOW_Lowes_ret_20d", "PFE_ret_1d", "AMD_ret_5d", "VVIX_ret_20d", "XOM_ret_1d", "XLY_Disc_vol_20d", "BA_ret_1d", "XLF_Fin_vol_20d", "US30Y_Rate_ret_20d", "HD_ret_1d", "LLY_zscore_60d", "heston_ev_h3"], "is_new": true}, {"model_id": "new_h7_NORMAL_RandomForest_N25_t0", "algo": "RandomForest", "regime": "NORMAL", "horizon": 7, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "T10Y2Y_Spread_ret_5d", "US1Y_Rate_ret_20d", "Brent_Oil_FRED_ret_5d", "TED_Spread_vol_20d", "AORD_AUS_zscore_60d", "EWY_Korea_zscore_60d", "AXP_Amex_ret_20d", "AMD_ret_5d", "VRP_ma5", "IYM_BasicMaterials_ret_20d", "M_Macys_vol_20d", "EWG_Germany_vol_20d", "US3M_Rate_vol_20d", "HangSeng_HK_ret_1d", "FedFunds_zscore_60d", "heston_ev_h3", "EWL_Switzerland_zscore_60d", "DOW_Price_zscore_60d", "EFFR_vol_20d", "CLX_Clorox_vol_20d", "SCHW_Schwab_ret_5d", "vix_mean_abs_ret_5d", "EXC_Exelon_zscore_60d"], "is_new": true}, {"model_id": "new_h7_NORMAL_RandomForest_N25_t1", "algo": "RandomForest", "regime": "NORMAL", "horizon": 7, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "NFCI_ret_5d", "CPB_CampbellSoup_ret_20d", "SBUX_zscore_60d", "LOW_Lowes_ret_5d", "US3M_Rate_zscore_60d", "Nikkei_Japan_zscore_60d", "EWA_Australia_ret_1d", "CI_Cigna_vol_20d", "EWG_Germany_vol_20d", "NEE_NextEra_ret_20d", "GILD_Gilead_ret_20d", "EWG_Germany_ret_20d", "US6M_Rate_ret_20d", "MS_MorganStanley_zscore_60d", "ES_Evergy_ret_1d", "CPB_CampbellSoup_zscore_60d", "XLF_Fin_vol_20d", "SCHW_Schwab_ret_5d", "MSTR_Bitcoin3_ret_20d", "FedFunds_zscore_60d", "BA_ret_1d", "EWA_Australia_zscore_60d", "EWM_Malaysia_zscore_60d"], "is_new": true}, {"model_id": "new_h7_NORMAL_RandomForest_N25_t2", "algo": "RandomForest", "regime": "NORMAL", "horizon": 7, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "heston_ev_h3", "MRK_Merck_zscore_60d", "BTI_BritishAmerican_ret_20d", "HD_ret_20d", "DIS_vol_20d", "CPB_CampbellSoup_ret_20d", "EXC_Exelon_ret_1d", "IYM_BasicMaterials_ret_20d", "HD_ret_5d", "ORCL_zscore_60d", "T10Y2Y_Spread_ret_5d", "spx_vol_5d", "NWL_Newell_ret_20d", "TED_Spread_zscore_60d", "VRP_ma5", "M_Macys_vol_20d", "LOW_Lowes_ret_20d", "LMT_LockheedMartin_ret_1d", "spx_abs_ret_max_5d", "AVB_AvalonBay_zscore_60d", "TM_Telephone_ret_1d", "MS_MorganStanley_ret_5d", "EQIX_Equinix_ret_5d"], "is_new": true}, {"model_id": "new_h7_NORMAL_RandomForest_N25_t3", "algo": "RandomForest", "regime": "NORMAL", "horizon": 7, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "US5Y_Rate_ret_5d", "GD_GeneralDynamics_zscore_60d", "DE_Deere_vol_20d", "Brent_Oil_FRED_ret_20d", "PAYX_Paychex_vol_20d", "AORD_AUS_zscore_60d", "AMD_ret_1d", "BA_ret_1d", "CPB_CampbellSoup_ret_20d", "TXN_vol_20d", "TED_Spread_zscore_60d", "US3M_Rate_vol_20d", "GILD_Gilead_ret_20d", "PG_ret_20d", "LLY_zscore_60d", "PAYX_Paychex_zscore_60d", "US30Y_Rate_ret_20d", "MS_MorganStanley_zscore_60d", "EMR_Emerson_ret_20d", "BTI_BritishAmerican_ret_20d", "SLB_Schlumberger_ret_1d", "AMZN_ret_5d", "gjr_condvar_h1"], "is_new": true}, {"model_id": "new_h7_NORMAL_RandomForest_N25_t4", "algo": "RandomForest", "regime": "NORMAL", "horizon": 7, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "ASX_Australia_vol_20d", "EOG_EOGResources_ret_5d", "3M_vol_20d", "US3Y_Rate_ret_5d", "NEE_NextEra_ret_20d", "DAX_Germany_zscore_60d", "T_ret_1d", "NWL_Newell_ret_20d", "US1Y_Rate_ret_20d", "Nikkei_Japan_vol_20d", "AXP_Amex_vol_20d", "EXC_Exelon_ret_1d", "GILD_Gilead_ret_20d", "3M_ret_5d", "XLB_Materials_zscore_60d", "WTI_Oil_FRED_zscore_60d", "VOD_Vodafone_zscore_60d", "SJM_JM_Smucker_ret_1d", "LMT_LockheedMartin_ret_1d", "PAYX_Paychex_zscore_60d", "BLK_BlackRock_zscore_60d", "Nikkei_Japan_zscore_60d", "Core_CPI_zscore_60d"], "is_new": true}, {"model_id": "new_h7_NORMAL_RandomForest_N25_t5", "algo": "RandomForest", "regime": "NORMAL", "horizon": 7, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "SLB_Schlumberger_ret_5d", "SJM_JM_Smucker_ret_5d", "XOM_ret_20d", "NVDA_vol_20d", "EOG_EOGResources_vol_20d", "AXP_Amex_vol_20d", "HangSeng_HK_vol_20d", "EWJ_Japan_vol_20d", "heston_var_ev_h3", "PAYX_Paychex_zscore_60d", "SO_SouthernCo_ret_5d", "gjr_condvar_h1", "CMCSA_ret_1d", "INTC_ret_5d", "TED_Spread_zscore_60d", "CPB_CampbellSoup_ret_20d", "HUM_Humana_ret_5d", "BTI_BritishAmerican_ret_20d", "AMGN_Amgen_ret_1d", "TM_Telephone_ret_1d", "T_ret_1d", "ES_Evergy_ret_1d", "MS_MorganStanley_ret_1d"], "is_new": true}, {"model_id": "new_h7_NORMAL_RandomForest_N25_t6", "algo": "RandomForest", "regime": "NORMAL", "horizon": 7, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "LOW_Lowes_ret_20d", "spx_momentum_3d", "NWL_Newell_ret_20d", "VVIX_ret_20d", "ASX_Australia_vol_20d", "PAYX_Paychex_ret_20d", "T_ret_1d", "EOG_EOGResources_ret_5d", "EWC_Canada_zscore_60d", "US3M_Rate_zscore_60d", "FedFunds_zscore_60d", "HangSeng_HK_ret_1d", "TED_Spread_zscore_60d", "AXP_Amex_vol_20d", "EXC_Exelon_ret_1d", "XOM_ret_1d", "Retail_Sales_zscore_60d", "EWJ_Japan_vol_20d", "GE_ret_1d", "AMD_ret_1d", "INTC_ret_1d", "PCAR_PaccarInc_ret_5d", "BTI_BritishAmerican_ret_5d"], "is_new": true}, {"model_id": "new_h7_NORMAL_RandomForest_N25_t7", "algo": "RandomForest", "regime": "NORMAL", "horizon": 7, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWG_Germany_vol_20d", "VOD_Vodafone_zscore_60d", "EXC_Exelon_zscore_60d", "Michigan_Sentiment_ret_20d", "EWG_Germany_ret_20d", "HD_ret_5d", "EWC_Canada_zscore_60d", "BTI_BritishAmerican_ret_20d", "MO_AltriaMG_ret_1d", "EQIX_Equinix_ret_5d", "AMD_ret_1d", "T_ret_1d", "ES_Evergy_ret_1d", "LUV_SouthwestAir_ret_5d", "SLB_Schlumberger_ret_5d", "AXP_Amex_vol_20d", "DE_Deere_ret_5d", "VRP_ma5", "US1Y_Rate_ret_20d", "DHR_ret_1d", "IYM_BasicMaterials_ret_20d", "CLX_Clorox_vol_20d", "NVDA_vol_20d"], "is_new": true}, {"model_id": "new_h7_NORMAL_RandomForest_N30_t0", "algo": "RandomForest", "regime": "NORMAL", "horizon": 7, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "PCAR_PaccarInc_ret_5d", "hmm_p_stress", "ES_Evergy_ret_1d", "PAYX_Paychex_ret_20d", "XLY_Disc_vol_20d", "MSTR_Bitcoin3_ret_1d", "M_Macys_vol_20d", "GILD_Gilead_ret_20d", "Brent_Oil_FRED_ret_20d", "3M_vol_20d", "AXP_Amex_vol_20d", "heston_var_ev_h3", "DE_Deere_vol_20d", "SLB_Schlumberger_ret_1d", "EWM_Malaysia_zscore_60d", "FedFunds_zscore_60d", "GE_ret_1d", "ENB_EnbridgeInc_ret_1d", "BLK_BlackRock_zscore_60d", "SBUX_vol_20d", "XLV_Health_zscore_60d", "BTI_BritishAmerican_ret_5d", "AMT_AmericanTower_ret_1d", "SLB_Schlumberger_ret_5d", "EWL_Switzerland_vol_20d", "VVIX_ret_20d", "US7Y_Rate_ret_20d", "TGT_Target_zscore_60d"], "is_new": true}, {"model_id": "new_h7_NORMAL_RandomForest_N30_t1", "algo": "RandomForest", "regime": "NORMAL", "horizon": 7, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "SCHW_Schwab_ret_5d", "vix_mean_abs_ret_5d", "XOM_ret_1d", "US5Y_Rate_ret_5d", "TGT_Target_zscore_60d", "INTC_ret_5d", "BTI_BritishAmerican_ret_20d", "XLK_Tech_zscore_60d", "MS_MorganStanley_zscore_60d", "LOW_Lowes_ret_20d", "EFFR_ret_1d", "TM_Telephone_vol_20d", "HUM_Humana_ret_5d", "IBEX_Spain_ret_20d", "IYM_BasicMaterials_ret_20d", "HD_ret_5d", "BLK_BlackRock_zscore_60d", "MSTR_Bitcoin3_ret_20d", "GILD_Gilead_ret_20d", "SBUX_zscore_60d", "PLD_Prologis_ret_5d", "ORCL_vol_20d", "XOM_ret_20d", "PPL_PPL_ret_1d", "US3Y_Rate_ret_5d", "BDX_Becton_Dickinson_ret_20d", "EWJ_Japan_vol_20d", "DIS_vol_20d"], "is_new": true}, {"model_id": "new_h7_NORMAL_RandomForest_N30_t2", "algo": "RandomForest", "regime": "NORMAL", "horizon": 7, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "US6M_Rate_ret_20d", "IWM_SmallCap_vol_20d", "AMD_ret_5d", "AMGN_Amgen_ret_1d", "AMD_ret_1d", "CPB_CampbellSoup_ret_5d", "LUV_SouthwestAir_ret_5d", "US3M_Rate_zscore_60d", "MS_MorganStanley_ret_1d", "VVIX_ret_20d", "EWH_HongKong_ret_5d", "LMT_LockheedMartin_vol_20d", "GD_GeneralDynamics_zscore_60d", "CI_Cigna_vol_20d", "VOD_Vodafone_zscore_60d", "SLB_Schlumberger_ret_1d", "US3Y_Rate_ret_5d", "gjr_condvar_h1", "EOG_EOGResources_vol_20d", "AVB_AvalonBay_zscore_60d", "US1Y_Rate_ret_5d", "US5Y_Rate_ret_5d", "EWY_Korea_ret_20d", "Nikkei_Japan_vol_20d", "ITT_ITTInc_ret_5d", "vix_acceleration_1d", "INTC_ret_1d", "EOG_EOGResources_ret_5d"], "is_new": true}, {"model_id": "new_h7_NORMAL_RandomForest_N30_t3", "algo": "RandomForest", "regime": "NORMAL", "horizon": 7, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "NEE_NextEra_ret_20d", "EXC_Exelon_zscore_60d", "EWG_Germany_ret_20d", "PAYX_Paychex_vol_20d", "PAYX_Paychex_zscore_60d", "Brent_Oil_FRED_ret_20d", "INTC_ret_5d", "PCAR_PaccarInc_ret_5d", "TM_Telephone_ret_1d", "VOD_Vodafone_zscore_60d", "US3M_Rate_zscore_60d", "EWM_Malaysia_zscore_60d", "SJM_JM_Smucker_ret_1d", "AXP_Amex_ret_20d", "EWA_Australia_zscore_60d", "TED_Spread_zscore_60d", "Retail_Sales_zscore_60d", "DHR_ret_1d", "MS_MorganStanley_ret_5d", "DE_Deere_vol_20d", "DAX_Germany_vol_20d", "PLD_Prologis_ret_5d", "SLB_Schlumberger_ret_1d", "T10Y2Y_Spread_ret_5d", "ENB_EnbridgeInc_ret_1d", "EWG_Germany_vol_20d", "Brent_Oil_FRED_ret_5d", "GD_GeneralDynamics_zscore_60d"], "is_new": true}, {"model_id": "new_h7_NORMAL_RandomForest_N30_t4", "algo": "RandomForest", "regime": "NORMAL", "horizon": 7, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "MS_MorganStanley_ret_1d", "WTI_Oil_FRED_zscore_60d", "spx_momentum_3d", "EWQ_France_zscore_60d", "DHR_vol_20d", "IYM_BasicMaterials_ret_20d", "PPL_PPL_ret_1d", "EQR_Equity_ret_1d", "MSTR_Bitcoin3_ret_1d", "AXP_Amex_vol_20d", "HD_ret_20d", "gjr_condvar_h1", "XLK_Tech_zscore_60d", "AORD_AUS_zscore_60d", "DE_Deere_vol_20d", "LMT_LockheedMartin_vol_20d", "DOW_Price_zscore_60d", "DAX_Germany_zscore_60d", "TED_Spread_zscore_60d", "AVB_AvalonBay_zscore_60d", "EWM_Malaysia_vol_20d", "US7Y_Rate_ret_20d", "ASX_Australia_ret_5d", "US3M_Rate_zscore_60d", "US30Y_Rate_ret_20d", "US1Y_Rate_ret_20d", "3M_ret_5d", "EFFR_ret_1d"], "is_new": true}, {"model_id": "new_h7_NORMAL_RandomForest_N30_t5", "algo": "RandomForest", "regime": "NORMAL", "horizon": 7, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWY_Korea_ret_20d", "HangSeng_HK_vol_20d", "US1Y_Rate_ret_20d", "HD_ret_20d", "PAYX_Paychex_ret_20d", "CPB_CampbellSoup_ret_5d", "DIS_vol_20d", "EXC_Exelon_zscore_60d", "HangSeng_HK_ret_5d", "US6M_Rate_ret_20d", "SBUX_ret_5d", "gjr_condvar_h1", "TM_Telephone_vol_20d", "SJM_JM_Smucker_ret_1d", "EQIX_Equinix_ret_5d", "Nikkei_Japan_zscore_60d", "SO_SouthernCo_ret_5d", "LOW_Lowes_ret_20d", "NFCI_ret_5d", "INTC_ret_1d", "LUV_SouthwestAir_ret_5d", "AMD_ret_5d", "XLB_Materials_zscore_60d", "DE_Deere_vol_20d", "SPY_zscore_60d", "HD_ret_1d", "EWQ_France_ret_20d", "AMD_ret_1d"], "is_new": true}, {"model_id": "new_h7_NORMAL_RandomForest_N30_t6", "algo": "RandomForest", "regime": "NORMAL", "horizon": 7, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "DOW_Price_zscore_60d", "GILD_Gilead_ret_20d", "NVDA_vol_20d", "FedFunds_zscore_60d", "LOW_Lowes_ret_5d", "EFFR_vol_20d", "hmm_p_stress", "AVB_AvalonBay_zscore_60d", "EWM_Malaysia_vol_20d", "XLK_Tech_zscore_60d", "HD_ret_5d", "US30Y_Rate_ret_20d", "heston_var_ev_h5", "LMT_LockheedMartin_vol_20d", "EWY_Korea_ret_20d", "XLV_Health_zscore_60d", "CPB_CampbellSoup_vol_20d", "TM_Telephone_ret_1d", "MRK_Merck_zscore_60d", "HUM_Humana_ret_5d", "EWS_Singapore_ret_5d", "SBUX_zscore_60d", "XLB_Materials_zscore_60d", "ES_Evergy_ret_1d", "NWL_Newell_ret_20d", "EOG_EOGResources_ret_5d", "ITT_ITTInc_ret_5d", "US3M_Rate_vol_20d"], "is_new": true}, {"model_id": "new_h7_NORMAL_RandomForest_N30_t7", "algo": "RandomForest", "regime": "NORMAL", "horizon": 7, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "ES_Evergy_ret_1d", "CLX_Clorox_vol_20d", "HUM_Humana_ret_5d", "XLB_Materials_zscore_60d", "LMT_LockheedMartin_ret_1d", "PCAR_PaccarInc_ret_5d", "XOM_ret_1d", "IYR_US_REIT2_zscore_60d", "Retail_Sales_zscore_60d", "MS_MorganStanley_ret_5d", "SCHW_Schwab_ret_5d", "M_Macys_vol_20d", "AMZN_ret_5d", "EOG_EOGResources_vol_20d", "HD_ret_20d", "XLV_Health_zscore_60d", "AXP_Amex_vol_20d", "Brent_Oil_FRED_ret_5d", "CPB_CampbellSoup_zscore_60d", "HD_ret_5d", "EWA_Australia_zscore_60d", "EQIX_Equinix_ret_5d", "XLF_Fin_vol_20d", "INTC_ret_5d", "GE_ret_1d", "TED_Spread_vol_20d", "SO_SouthernCo_ret_5d", "3M_vol_20d"], "is_new": true}, {"model_id": "new_h7_NORMAL_LogisticRegression_N5_t0", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 7, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "SBUX_zscore_60d", "US1Y_Rate_ret_5d", "T10Y2Y_Spread_ret_5d"], "is_new": true}, {"model_id": "new_h7_NORMAL_LogisticRegression_N5_t1", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 7, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "LMT_LockheedMartin_ret_1d", "AXP_Amex_vol_20d", "LUV_SouthwestAir_ret_5d"], "is_new": true}, {"model_id": "new_h7_NORMAL_LogisticRegression_N5_t2", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 7, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "QQQ_vol_20d", "EWY_Korea_zscore_60d", "TGT_Target_zscore_60d"], "is_new": true}, {"model_id": "new_h7_NORMAL_LogisticRegression_N5_t3", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 7, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "LUV_SouthwestAir_ret_5d", "IYR_US_REIT2_zscore_60d", "HD_ret_5d"], "is_new": true}, {"model_id": "new_h7_NORMAL_LogisticRegression_N5_t4", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 7, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "IBEX_Spain_ret_20d", "SBUX_vol_20d", "ES_Evergy_ret_1d"], "is_new": true}, {"model_id": "new_h7_NORMAL_LogisticRegression_N5_t5", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 7, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EFFR_vol_20d", "SCHW_Schwab_ret_5d", "LUV_SouthwestAir_ret_5d"], "is_new": true}, {"model_id": "new_h7_NORMAL_LogisticRegression_N5_t6", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 7, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "SLB_Schlumberger_ret_1d", "US3M_Rate_vol_20d", "EWM_Malaysia_ret_1d"], "is_new": true}, {"model_id": "new_h7_NORMAL_LogisticRegression_N5_t7", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 7, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "US3Y_Rate_ret_5d", "MSTR_Bitcoin3_ret_5d", "EQR_Equity_ret_1d"], "is_new": true}, {"model_id": "new_h7_NORMAL_LogisticRegression_N8_t0", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 7, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "CI_Cigna_vol_20d", "CPB_CampbellSoup_vol_20d", "EQR_Equity_ret_1d", "CPB_CampbellSoup_zscore_60d", "EWY_Korea_ret_20d", "US3M_Rate_zscore_60d"], "is_new": true}, {"model_id": "new_h7_NORMAL_LogisticRegression_N8_t1", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 7, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "QQQ_vol_20d", "EWG_Germany_ret_20d", "PAYX_Paychex_ret_20d", "EXC_Exelon_zscore_60d", "vix_acceleration_1d", "XLB_Materials_zscore_60d"], "is_new": true}, {"model_id": "new_h7_NORMAL_LogisticRegression_N8_t2", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 7, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "AXP_Amex_ret_20d", "EOG_EOGResources_vol_20d", "CPB_CampbellSoup_ret_20d", "IYR_US_REIT2_zscore_60d", "GD_GeneralDynamics_zscore_60d", "HangSeng_HK_vol_20d"], "is_new": true}, {"model_id": "new_h7_NORMAL_LogisticRegression_N8_t3", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 7, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "PAYX_Paychex_ret_20d", "IYM_BasicMaterials_ret_20d", "TM_Telephone_vol_20d", "GE_ret_1d", "EWG_Germany_vol_20d", "PLD_Prologis_ret_5d"], "is_new": true}, {"model_id": "new_h7_NORMAL_LogisticRegression_N8_t4", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 7, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "spx_abs_ret_max_5d", "TXN_vol_20d", "AMGN_Amgen_ret_1d", "MSTR_Bitcoin3_ret_20d", "heston_ev_h3", "DAX_Germany_zscore_60d"], "is_new": true}, {"model_id": "new_h7_NORMAL_LogisticRegression_N8_t5", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 7, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "ORCL_zscore_60d", "Industrial_Production_zscore_60d", "CPB_CampbellSoup_ret_5d", "LOW_Lowes_ret_5d", "US1Y_Rate_ret_5d", "XOM_ret_1d"], "is_new": true}, {"model_id": "new_h7_NORMAL_LogisticRegression_N8_t6", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 7, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "BA_ret_1d", "EXC_Exelon_zscore_60d", "LMT_LockheedMartin_vol_20d", "spx_abs_ret_max_5d", "CPB_CampbellSoup_zscore_60d", "EWY_Korea_zscore_60d"], "is_new": true}, {"model_id": "new_h7_NORMAL_LogisticRegression_N8_t7", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 7, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWH_HongKong_ret_5d", "Retail_Sales_zscore_60d", "CPB_CampbellSoup_ret_20d", "ASX_Australia_vol_20d", "US6M_Rate_ret_20d", "US7Y_Rate_ret_20d"], "is_new": true}, {"model_id": "new_h7_NORMAL_LogisticRegression_N10_t0", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 7, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "ENB_EnbridgeInc_ret_1d", "EXC_Exelon_ret_1d", "VOD_Vodafone_zscore_60d", "LUV_SouthwestAir_ret_5d", "MS_MorganStanley_ret_1d", "TM_Telephone_ret_1d", "Brent_Oil_FRED_ret_20d", "VVIX_ret_20d"], "is_new": true}, {"model_id": "new_h7_NORMAL_LogisticRegression_N10_t1", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 7, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "US6M_Rate_ret_20d", "Brent_Oil_FRED_ret_5d", "VRP_ma5", "PAYX_Paychex_zscore_60d", "GD_GeneralDynamics_zscore_60d", "DIS_vol_20d", "Nikkei_Japan_vol_20d", "XLV_Health_zscore_60d"], "is_new": true}, {"model_id": "new_h7_NORMAL_LogisticRegression_N10_t2", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 7, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "PAYX_Paychex_zscore_60d", "DE_Deere_vol_20d", "EWH_HongKong_ret_5d", "EWY_Korea_zscore_60d", "spx_abs_ret_max_5d", "NWL_Newell_ret_20d", "MS_MorganStanley_ret_5d", "AMGN_Amgen_ret_1d"], "is_new": true}, {"model_id": "new_h7_NORMAL_LogisticRegression_N10_t3", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 7, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWQ_France_ret_20d", "CPB_CampbellSoup_ret_5d", "SCHW_Schwab_ret_5d", "CPB_CampbellSoup_zscore_60d", "T10Y2Y_Spread_ret_5d", "Retail_Sales_zscore_60d", "CMCSA_ret_1d", "TED_Spread_zscore_60d"], "is_new": true}, {"model_id": "new_h7_NORMAL_LogisticRegression_N10_t4", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 7, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "Brent_Oil_FRED_ret_5d", "EWM_Malaysia_ret_1d", "FedFunds_zscore_60d", "PAYX_Paychex_vol_20d", "XLF_Fin_vol_20d", "EFFR_ret_1d", "CPB_CampbellSoup_zscore_60d", "US3Y_Rate_ret_5d"], "is_new": true}, {"model_id": "new_h7_NORMAL_LogisticRegression_N10_t5", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 7, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "gjr_condvar_h1", "CTAS_Cintas_vol_20d", "SBUX_ret_5d", "CCI_CrownCastle_vol_20d", "SO_SouthernCo_ret_5d", "INTC_ret_1d", "MS_MorganStanley_ret_1d", "ASX_Australia_ret_5d"], "is_new": true}, {"model_id": "new_h7_NORMAL_LogisticRegression_N10_t6", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 7, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "TGT_Target_zscore_60d", "heston_var_ev_h3", "XOM_ret_1d", "TED_Spread_vol_20d", "BTI_BritishAmerican_ret_5d", "LUV_SouthwestAir_ret_5d", "EWM_Malaysia_zscore_60d", "ES_Evergy_ret_1d"], "is_new": true}, {"model_id": "new_h7_NORMAL_LogisticRegression_N10_t7", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 7, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWM_Malaysia_vol_20d", "LOW_Lowes_ret_5d", "LOW_Lowes_ret_20d", "SO_SouthernCo_ret_5d", "EWC_Canada_zscore_60d", "Brent_Oil_FRED_ret_5d", "AMZN_ret_5d", "ORCL_vol_20d"], "is_new": true}, {"model_id": "new_h7_NORMAL_LogisticRegression_N12_t0", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 7, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "NFCI_ret_5d", "VRP_ma5", "HangSeng_HK_ret_1d", "ASX_Australia_vol_20d", "EWY_Korea_zscore_60d", "US3Y_Rate_ret_5d", "MSTR_Bitcoin3_ret_1d", "EWQ_France_zscore_60d", "IYR_US_REIT2_zscore_60d", "AMT_AmericanTower_ret_1d"], "is_new": true}, {"model_id": "new_h7_NORMAL_LogisticRegression_N12_t1", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 7, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "CPB_CampbellSoup_ret_5d", "Core_CPI_zscore_60d", "US1Y_Rate_ret_20d", "NVDA_vol_20d", "AMD_ret_1d", "XLB_Materials_zscore_60d", "BLK_BlackRock_zscore_60d", "US6M_Rate_ret_20d", "hmm_p_stress", "CTAS_Cintas_vol_20d"], "is_new": true}, {"model_id": "new_h7_NORMAL_LogisticRegression_N12_t2", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 7, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "JNJ_ret_1d", "heston_var_ev_h5", "BDX_Becton_Dickinson_ret_20d", "US1Y_Rate_ret_5d", "LUV_SouthwestAir_ret_5d", "Michigan_Sentiment_ret_20d", "DHR_vol_20d", "EWY_Korea_zscore_60d", "IWM_SmallCap_vol_20d", "Industrial_Production_zscore_60d"], "is_new": true}, {"model_id": "new_h7_NORMAL_LogisticRegression_N12_t3", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 7, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "XLV_Health_zscore_60d", "ASX_Australia_vol_20d", "GD_GeneralDynamics_zscore_60d", "EFFR_ret_1d", "US3M_Rate_zscore_60d", "SLB_Schlumberger_ret_1d", "LOW_Lowes_ret_5d", "MS_MorganStanley_zscore_60d", "INTC_ret_1d", "T_ret_1d"], "is_new": true}, {"model_id": "new_h7_NORMAL_LogisticRegression_N12_t4", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 7, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "XLV_Health_zscore_60d", "SBUX_vol_20d", "EMR_Emerson_ret_20d", "SBUX_zscore_60d", "ITT_ITTInc_ret_5d", "EWY_Korea_zscore_60d", "LMT_LockheedMartin_vol_20d", "TED_Spread_vol_20d", "CCI_CrownCastle_vol_20d", "HD_ret_20d"], "is_new": true}, {"model_id": "new_h7_NORMAL_LogisticRegression_N12_t5", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 7, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "HD_ret_5d", "LMT_LockheedMartin_ret_1d", "XLF_Fin_vol_20d", "VRP_ma5", "US3M_Rate_zscore_60d", "EFFR_ret_1d", "SBUX_vol_20d", "GE_ret_1d", "M_Macys_vol_20d", "vix_acceleration_1d"], "is_new": true}, {"model_id": "new_h7_NORMAL_LogisticRegression_N12_t6", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 7, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "NEE_NextEra_ret_20d", "QQQ_vol_20d", "TGT_Target_zscore_60d", "INTC_ret_5d", "LMT_LockheedMartin_vol_20d", "IYM_BasicMaterials_ret_20d", "PAYX_Paychex_vol_20d", "NOC_Northrop_ret_20d", "3M_vol_20d", "SLB_Schlumberger_ret_1d"], "is_new": true}, {"model_id": "new_h7_NORMAL_LogisticRegression_N12_t7", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 7, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWJ_Japan_vol_20d", "XLF_Fin_vol_20d", "AMD_ret_1d", "EFFR_ret_1d", "XLB_Materials_zscore_60d", "MS_MorganStanley_ret_1d", "GILD_Gilead_ret_20d", "ASX_Australia_ret_5d", "SO_SouthernCo_ret_5d", "DAX_Germany_zscore_60d"], "is_new": true}, {"model_id": "new_h7_NORMAL_LogisticRegression_N15_t0", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 7, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "Michigan_Sentiment_ret_20d", "GE_ret_1d", "IYR_US_REIT2_zscore_60d", "VVIX_ret_20d", "US7Y_Rate_ret_20d", "Nikkei_Japan_vol_20d", "EWM_Malaysia_zscore_60d", "CMCSA_ret_1d", "TGT_Target_zscore_60d", "TED_Spread_vol_20d", "SBUX_zscore_60d", "heston_var_ev_h3", "Core_CPI_zscore_60d"], "is_new": true}, {"model_id": "new_h7_NORMAL_LogisticRegression_N15_t1", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 7, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "LLY_zscore_60d", "CTAS_Cintas_vol_20d", "VOD_Vodafone_zscore_60d", "ORCL_vol_20d", "AMT_AmericanTower_ret_1d", "US3M_Rate_zscore_60d", "MO_AltriaMG_ret_1d", "SLB_Schlumberger_ret_1d", "MS_MorganStanley_ret_1d", "BLK_BlackRock_zscore_60d", "IYR_US_REIT2_zscore_60d", "AMGN_Amgen_ret_1d", "LMT_LockheedMartin_vol_20d"], "is_new": true}, {"model_id": "new_h7_NORMAL_LogisticRegression_N15_t2", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 7, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "US6M_Rate_ret_20d", "heston_var_ev_h7", "CPB_CampbellSoup_vol_20d", "EFFR_ret_1d", "ES_Evergy_ret_1d", "MO_AltriaMG_ret_1d", "Core_CPI_zscore_60d", "IYR_US_REIT2_zscore_60d", "Retail_Sales_zscore_60d", "JNJ_ret_1d", "EWL_Switzerland_vol_20d", "SO_SouthernCo_ret_5d", "AVB_AvalonBay_zscore_60d"], "is_new": true}, {"model_id": "new_h7_NORMAL_LogisticRegression_N15_t3", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 7, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "US3M_Rate_zscore_60d", "BLK_BlackRock_zscore_60d", "DE_Deere_ret_5d", "TED_Spread_zscore_60d", "VRP_ma5", "CPB_CampbellSoup_ret_20d", "EXC_Exelon_ret_1d", "CPB_CampbellSoup_ret_5d", "CPB_CampbellSoup_zscore_60d", "AXP_Amex_ret_20d", "WTI_Oil_FRED_zscore_60d", "PLD_Prologis_ret_5d", "XLF_Fin_vol_20d"], "is_new": true}, {"model_id": "new_h7_NORMAL_LogisticRegression_N15_t4", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 7, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EFFR_vol_20d", "INTC_ret_5d", "US30Y_Rate_ret_20d", "HangSeng_HK_ret_1d", "ASX_Australia_ret_5d", "HD_zscore_60d", "AMGN_Amgen_ret_1d", "US6M_Rate_ret_20d", "EWG_Germany_ret_20d", "EWY_Korea_ret_20d", "ORCL_zscore_60d", "EWG_Germany_vol_20d", "EOG_EOGResources_vol_20d"], "is_new": true}, {"model_id": "new_h7_NORMAL_LogisticRegression_N15_t5", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 7, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "TM_Telephone_ret_1d", "PAYX_Paychex_ret_20d", "EQR_Equity_ret_1d", "EWG_Germany_ret_20d", "US1Y_Rate_ret_5d", "MO_AltriaMG_ret_1d", "Michigan_Sentiment_ret_20d", "IYM_BasicMaterials_ret_20d", "NVDA_vol_20d", "HUM_Humana_ret_5d", "TED_Spread_zscore_60d", "NWL_Newell_ret_20d", "DHR_ret_1d"], "is_new": true}, {"model_id": "new_h7_NORMAL_LogisticRegression_N15_t6", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 7, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "MSTR_Bitcoin3_ret_20d", "XLF_Fin_vol_20d", "Industrial_Production_zscore_60d", "SPY_zscore_60d", "AVB_AvalonBay_zscore_60d", "AMD_ret_5d", "heston_var_ev_h5", "EWY_Korea_zscore_60d", "T_ret_1d", "spx_vol_5d", "WTI_Oil_FRED_zscore_60d", "CMCSA_ret_1d", "NEE_NextEra_ret_20d"], "is_new": true}, {"model_id": "new_h7_NORMAL_LogisticRegression_N15_t7", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 7, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "NOC_Northrop_ret_20d", "LLY_zscore_60d", "hmm_p_stress", "Core_CPI_zscore_60d", "CI_Cigna_vol_20d", "Nikkei_Japan_zscore_60d", "spx_vol_5d", "EWC_Canada_zscore_60d", "XOM_ret_1d", "XLY_Disc_vol_20d", "EWG_Germany_ret_20d", "CPB_CampbellSoup_vol_20d", "US3M_Rate_vol_20d"], "is_new": true}, {"model_id": "new_h7_NORMAL_LogisticRegression_N20_t0", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 7, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "spx_momentum_3d", "spx_vol_5d", "SLB_Schlumberger_ret_1d", "M_Macys_vol_20d", "SLB_Schlumberger_ret_5d", "XLY_Disc_vol_20d", "EWQ_France_zscore_60d", "XOM_ret_1d", "SBUX_ret_5d", "SCHW_Schwab_ret_5d", "PAYX_Paychex_ret_20d", "BA_ret_1d", "HangSeng_HK_ret_1d", "ES_Evergy_ret_1d", "LMT_LockheedMartin_ret_1d", "spx_abs_ret_max_5d", "DAX_Germany_vol_20d", "US3M_Rate_zscore_60d"], "is_new": true}, {"model_id": "new_h7_NORMAL_LogisticRegression_N20_t1", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 7, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWQ_France_ret_20d", "AMZN_ret_5d", "NOC_Northrop_ret_20d", "XLF_Fin_vol_20d", "LUV_SouthwestAir_ret_5d", "NVDA_vol_20d", "XLK_Tech_zscore_60d", "HUM_Humana_ret_5d", "MO_AltriaMG_ret_1d", "WTI_Oil_FRED_zscore_60d", "SCHW_Schwab_ret_5d", "HangSeng_HK_vol_20d", "heston_var_ev_h5", "Core_CPI_zscore_60d", "IBEX_Spain_ret_20d", "ITT_ITTInc_ret_5d", "CPB_CampbellSoup_zscore_60d", "TED_Spread_zscore_60d"], "is_new": true}, {"model_id": "new_h7_NORMAL_LogisticRegression_N20_t2", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 7, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "CPB_CampbellSoup_zscore_60d", "MRK_Merck_zscore_60d", "SO_SouthernCo_ret_5d", "LLY_zscore_60d", "HD_ret_20d", "EWQ_France_zscore_60d", "HD_zscore_60d", "Industrial_Production_zscore_60d", "US7Y_Rate_ret_20d", "EQIX_Equinix_ret_5d", "AMD_ret_5d", "EFFR_ret_1d", "EWS_Singapore_ret_5d", "HUM_Humana_ret_5d", "US1Y_Rate_ret_5d", "DOW_Price_zscore_60d", "BDX_Becton_Dickinson_ret_20d", "EOG_EOGResources_vol_20d"], "is_new": true}, {"model_id": "new_h7_NORMAL_LogisticRegression_N20_t3", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 7, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "SJM_JM_Smucker_ret_5d", "EWQ_France_ret_20d", "EQR_Equity_ret_1d", "SO_SouthernCo_ret_5d", "BDX_Becton_Dickinson_ret_20d", "US1Y_Rate_ret_20d", "gjr_condvar_h1", "PLD_Prologis_ret_5d", "EWJ_Japan_vol_20d", "ITT_ITTInc_ret_5d", "EWM_Malaysia_zscore_60d", "spx_abs_ret_max_5d", "WTI_Oil_FRED_zscore_60d", "EXC_Exelon_ret_1d", "DAX_Germany_vol_20d", "CPB_CampbellSoup_zscore_60d", "PAYX_Paychex_ret_20d", "ORCL_zscore_60d"], "is_new": true}, {"model_id": "new_h7_NORMAL_LogisticRegression_N20_t4", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 7, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "T10Y2Y_Spread_ret_5d", "SBUX_zscore_60d", "JNJ_ret_1d", "T_ret_1d", "MSTR_Bitcoin3_ret_5d", "hmm_p_stress", "EQIX_Equinix_ret_5d", "US1Y_Rate_ret_5d", "IWM_SmallCap_vol_20d", "QQQ_vol_20d", "GE_ret_1d", "EWA_Australia_zscore_60d", "EWH_HongKong_ret_5d", "CPB_CampbellSoup_ret_20d", "CTAS_Cintas_vol_20d", "NOC_Northrop_ret_20d", "HD_ret_1d", "PAYX_Paychex_vol_20d"], "is_new": true}, {"model_id": "new_h7_NORMAL_LogisticRegression_N20_t5", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 7, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "IYR_US_REIT2_zscore_60d", "EWA_Australia_ret_1d", "Nikkei_Japan_vol_20d", "AMT_AmericanTower_ret_1d", "SBUX_zscore_60d", "HD_ret_20d", "EWQ_France_ret_20d", "US5Y_Rate_ret_5d", "TM_Telephone_vol_20d", "US7Y_Rate_ret_20d", "US3M_Rate_vol_20d", "TGT_Target_zscore_60d", "ASX_Australia_vol_20d", "XLY_Disc_vol_20d", "3M_vol_20d", "PLD_Prologis_ret_5d", "LLY_zscore_60d", "AXP_Amex_vol_20d"], "is_new": true}, {"model_id": "new_h7_NORMAL_LogisticRegression_N20_t6", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 7, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "LUV_SouthwestAir_ret_5d", "US3M_Rate_zscore_60d", "US3Y_Rate_ret_5d", "XOM_ret_1d", "T_ret_1d", "hmm_p_stress", "SJM_JM_Smucker_ret_5d", "QQQ_vol_20d", "3M_vol_20d", "GD_GeneralDynamics_zscore_60d", "Michigan_Sentiment_ret_20d", "DAX_Germany_zscore_60d", "TGT_Target_zscore_60d", "BTI_BritishAmerican_ret_5d", "IYM_BasicMaterials_ret_20d", "XLB_Materials_zscore_60d", "EWY_Korea_ret_20d", "EWJ_Japan_vol_20d"], "is_new": true}, {"model_id": "new_h7_NORMAL_LogisticRegression_N20_t7", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 7, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "vix_mean_abs_ret_5d", "CPB_CampbellSoup_zscore_60d", "NFCI_ret_5d", "3M_vol_20d", "HD_ret_1d", "CI_Cigna_vol_20d", "MSTR_Bitcoin3_ret_5d", "MS_MorganStanley_ret_1d", "SLB_Schlumberger_ret_5d", "CCI_CrownCastle_vol_20d", "spx_vol_5d", "EWY_Korea_ret_20d", "JNJ_ret_1d", "SBUX_zscore_60d", "Nikkei_Japan_vol_20d", "EWQ_France_zscore_60d", "Brent_Oil_FRED_ret_20d", "Core_CPI_zscore_60d"], "is_new": true}, {"model_id": "new_h7_NORMAL_LogisticRegression_N25_t0", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 7, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "SJM_JM_Smucker_ret_5d", "EWY_Korea_ret_20d", "CLX_Clorox_vol_20d", "NEE_NextEra_ret_20d", "ASX_Australia_vol_20d", "SLB_Schlumberger_ret_5d", "TM_Telephone_ret_1d", "PLD_Prologis_ret_5d", "HangSeng_HK_ret_1d", "LOW_Lowes_ret_5d", "EWQ_France_zscore_60d", "LLY_zscore_60d", "CI_Cigna_vol_20d", "T10Y2Y_Spread_ret_5d", "PAYX_Paychex_zscore_60d", "US1Y_Rate_ret_20d", "Nikkei_Japan_vol_20d", "EWJ_Japan_vol_20d", "SPY_zscore_60d", "Industrial_Production_zscore_60d", "heston_var_ev_h7", "HD_ret_5d", "PCAR_PaccarInc_ret_5d"], "is_new": true}, {"model_id": "new_h7_NORMAL_LogisticRegression_N25_t1", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 7, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "US6M_Rate_ret_20d", "LLY_zscore_60d", "AXP_Amex_ret_20d", "EWL_Switzerland_vol_20d", "ITT_ITTInc_ret_5d", "vix_acceleration_1d", "SO_SouthernCo_ret_5d", "US30Y_Rate_ret_20d", "XLY_Disc_vol_20d", "AVB_AvalonBay_zscore_60d", "Industrial_Production_zscore_60d", "spx_momentum_3d", "EWA_Australia_zscore_60d", "EMR_Emerson_ret_20d", "INTC_ret_5d", "IYR_US_REIT2_zscore_60d", "SCHW_Schwab_ret_5d", "hmm_p_stress", "Retail_Sales_zscore_60d", "IBEX_Spain_ret_20d", "TXN_vol_20d", "VRP_ma5", "SBUX_vol_20d"], "is_new": true}, {"model_id": "new_h7_NORMAL_LogisticRegression_N25_t2", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 7, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EQR_Equity_ret_1d", "EQIX_Equinix_ret_5d", "AVB_AvalonBay_zscore_60d", "EMR_Emerson_ret_20d", "HangSeng_HK_ret_5d", "HD_ret_1d", "IYR_US_REIT2_zscore_60d", "T10Y2Y_Spread_ret_5d", "XLK_Tech_zscore_60d", "BA_ret_1d", "EOG_EOGResources_ret_5d", "EWL_Switzerland_zscore_60d", "SBUX_zscore_60d", "ASX_Australia_ret_5d", "DE_Deere_ret_5d", "GD_GeneralDynamics_zscore_60d", "AORD_AUS_zscore_60d", "DIS_vol_20d", "EXC_Exelon_zscore_60d", "CTAS_Cintas_vol_20d", "SLB_Schlumberger_ret_5d", "SBUX_ret_5d", "US3Y_Rate_ret_5d"], "is_new": true}, {"model_id": "new_h7_NORMAL_LogisticRegression_N25_t3", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 7, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "TED_Spread_vol_20d", "MS_MorganStanley_ret_1d", "US1Y_Rate_ret_5d", "ORCL_vol_20d", "SCHW_Schwab_ret_5d", "Core_CPI_zscore_60d", "EXC_Exelon_ret_1d", "US5Y_Rate_ret_5d", "CCI_CrownCastle_vol_20d", "Nikkei_Japan_zscore_60d", "US7Y_Rate_ret_20d", "MS_MorganStanley_ret_5d", "spx_vol_5d", "AMD_ret_5d", "CLX_Clorox_vol_20d", "LUV_SouthwestAir_ret_5d", "heston_var_ev_h5", "XOM_ret_1d", "XLB_Materials_zscore_60d", "IWM_SmallCap_vol_20d", "CPB_CampbellSoup_ret_5d", "3M_ret_5d", "Brent_Oil_FRED_ret_20d"], "is_new": true}, {"model_id": "new_h7_NORMAL_LogisticRegression_N25_t4", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 7, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "LOW_Lowes_ret_5d", "gjr_condvar_h1", "ES_Evergy_ret_1d", "NWL_Newell_ret_20d", "HD_ret_1d", "AMGN_Amgen_ret_1d", "INTC_ret_1d", "Brent_Oil_FRED_ret_20d", "SBUX_vol_20d", "XLV_Health_zscore_60d", "IYR_US_REIT2_zscore_60d", "SPY_zscore_60d", "US3M_Rate_vol_20d", "DE_Deere_vol_20d", "BTI_BritishAmerican_ret_5d", "XLK_Tech_zscore_60d", "CPB_CampbellSoup_ret_5d", "QQQ_vol_20d", "LUV_SouthwestAir_ret_5d", "US6M_Rate_ret_20d", "LOW_Lowes_ret_20d", "DIS_vol_20d", "CPB_CampbellSoup_zscore_60d"], "is_new": true}, {"model_id": "new_h7_NORMAL_LogisticRegression_N25_t5", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 7, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "XLF_Fin_vol_20d", "SJM_JM_Smucker_ret_1d", "JNJ_ret_1d", "gjr_condvar_h1", "XLB_Materials_zscore_60d", "CPB_CampbellSoup_vol_20d", "PAYX_Paychex_ret_20d", "QQQ_vol_20d", "NEE_NextEra_ret_20d", "NVDA_vol_20d", "IBEX_Spain_ret_20d", "ASX_Australia_vol_20d", "BA_ret_1d", "heston_var_ev_h7", "DE_Deere_ret_5d", "AORD_AUS_zscore_60d", "EWH_HongKong_ret_5d", "ENB_EnbridgeInc_ret_1d", "EFFR_ret_1d", "spx_vol_5d", "TM_Telephone_ret_1d", "MO_AltriaMG_ret_1d", "PAYX_Paychex_vol_20d"], "is_new": true}, {"model_id": "new_h7_NORMAL_LogisticRegression_N25_t6", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 7, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "CPB_CampbellSoup_vol_20d", "DHR_vol_20d", "Core_CPI_zscore_60d", "Industrial_Production_zscore_60d", "PPL_PPL_ret_1d", "TED_Spread_zscore_60d", "LMT_LockheedMartin_vol_20d", "MRK_Merck_zscore_60d", "BA_ret_1d", "HangSeng_HK_vol_20d", "hmm_p_stress", "MS_MorganStanley_ret_5d", "T_ret_1d", "heston_var_ev_h3", "HD_zscore_60d", "TXN_vol_20d", "PCAR_PaccarInc_ret_5d", "JNJ_ret_1d", "AMZN_ret_5d", "XLV_Health_zscore_60d", "DE_Deere_ret_5d", "CPB_CampbellSoup_zscore_60d", "MSTR_Bitcoin3_ret_5d"], "is_new": true}, {"model_id": "new_h7_NORMAL_LogisticRegression_N25_t7", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 7, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "PPL_PPL_ret_1d", "MSTR_Bitcoin3_ret_5d", "PAYX_Paychex_zscore_60d", "XLB_Materials_zscore_60d", "ORCL_zscore_60d", "SBUX_zscore_60d", "CI_Cigna_vol_20d", "SBUX_ret_5d", "IYM_BasicMaterials_ret_20d", "spx_momentum_3d", "EWQ_France_ret_20d", "ASX_Australia_ret_5d", "AORD_AUS_zscore_60d", "MSTR_Bitcoin3_ret_20d", "Core_PCE_zscore_60d", "PCAR_PaccarInc_ret_5d", "SJM_JM_Smucker_ret_1d", "TM_Telephone_ret_1d", "HD_ret_1d", "HangSeng_HK_vol_20d", "EXC_Exelon_ret_1d", "MO_AltriaMG_ret_1d", "CLX_Clorox_vol_20d"], "is_new": true}, {"model_id": "new_h7_NORMAL_LogisticRegression_N30_t0", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 7, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "BLK_BlackRock_zscore_60d", "EWG_Germany_ret_20d", "MS_MorganStanley_ret_1d", "SLB_Schlumberger_ret_5d", "SBUX_vol_20d", "EWM_Malaysia_ret_1d", "EWS_Singapore_ret_5d", "VVIX_ret_20d", "Brent_Oil_FRED_ret_5d", "CPB_CampbellSoup_zscore_60d", "IYR_US_REIT2_zscore_60d", "US1Y_Rate_ret_5d", "EWY_Korea_zscore_60d", "EWG_Germany_vol_20d", "AXP_Amex_ret_20d", "Core_CPI_zscore_60d", "EWM_Malaysia_zscore_60d", "BTI_BritishAmerican_ret_20d", "heston_var_ev_h7", "HD_ret_1d", "US1Y_Rate_ret_20d", "Retail_Sales_zscore_60d", "HD_ret_5d", "EQR_Equity_ret_1d", "DE_Deere_ret_5d", "SCHW_Schwab_ret_5d", "GE_ret_1d", "EWL_Switzerland_vol_20d"], "is_new": true}, {"model_id": "new_h7_NORMAL_LogisticRegression_N30_t1", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 7, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "HD_zscore_60d", "EWM_Malaysia_vol_20d", "VOD_Vodafone_zscore_60d", "TED_Spread_vol_20d", "GE_ret_1d", "AMGN_Amgen_ret_1d", "Core_CPI_zscore_60d", "AMZN_ret_5d", "HangSeng_HK_vol_20d", "US30Y_Rate_ret_20d", "T10Y2Y_Spread_ret_5d", "AMD_ret_1d", "PAYX_Paychex_zscore_60d", "heston_var_ev_h3", "GD_GeneralDynamics_zscore_60d", "TED_Spread_zscore_60d", "SJM_JM_Smucker_ret_5d", "BDX_Becton_Dickinson_ret_20d", "CTAS_Cintas_vol_20d", "EQIX_Equinix_ret_5d", "PFE_ret_1d", "NWL_Newell_ret_20d", "EWM_Malaysia_zscore_60d", "XOM_ret_1d", "XLF_Fin_vol_20d", "MS_MorganStanley_ret_5d", "EOG_EOGResources_vol_20d", "US1Y_Rate_ret_20d"], "is_new": true}, {"model_id": "new_h7_NORMAL_LogisticRegression_N30_t2", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 7, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "heston_ev_h3", "XLK_Tech_zscore_60d", "XOM_ret_20d", "XLV_Health_zscore_60d", "HangSeng_HK_ret_1d", "DHR_ret_1d", "GE_ret_1d", "BTI_BritishAmerican_ret_5d", "HangSeng_HK_vol_20d", "LMT_LockheedMartin_vol_20d", "EQR_Equity_ret_1d", "US7Y_Rate_ret_20d", "US3M_Rate_zscore_60d", "VOD_Vodafone_zscore_60d", "spx_vol_5d", "CI_Cigna_vol_20d", "EXC_Exelon_ret_1d", "EWA_Australia_ret_1d", "3M_vol_20d", "BLK_BlackRock_zscore_60d", "GD_GeneralDynamics_zscore_60d", "AORD_AUS_zscore_60d", "NEE_NextEra_ret_20d", "XLB_Materials_zscore_60d", "T10Y2Y_Spread_ret_5d", "TXN_vol_20d", "NOC_Northrop_ret_20d", "SLB_Schlumberger_ret_1d"], "is_new": true}, {"model_id": "new_h7_NORMAL_LogisticRegression_N30_t3", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 7, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "CLX_Clorox_vol_20d", "SO_SouthernCo_ret_5d", "MS_MorganStanley_ret_5d", "Industrial_Production_zscore_60d", "MSTR_Bitcoin3_ret_1d", "CTAS_Cintas_vol_20d", "EWY_Korea_zscore_60d", "AMT_AmericanTower_ret_1d", "EWA_Australia_ret_1d", "IWM_SmallCap_vol_20d", "GE_ret_1d", "FedFunds_zscore_60d", "AMZN_ret_5d", "Michigan_Sentiment_ret_20d", "AORD_AUS_zscore_60d", "heston_var_ev_h5", "spx_momentum_3d", "EWJ_Japan_vol_20d", "ES_Evergy_ret_1d", "LOW_Lowes_ret_20d", "US1Y_Rate_ret_5d", "PPL_PPL_ret_1d", "HUM_Humana_ret_5d", "TM_Telephone_ret_1d", "M_Macys_vol_20d", "SCHW_Schwab_ret_5d", "BTI_BritishAmerican_ret_5d", "EMR_Emerson_ret_20d"], "is_new": true}, {"model_id": "new_h7_NORMAL_LogisticRegression_N30_t4", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 7, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "MO_AltriaMG_ret_1d", "PLD_Prologis_ret_5d", "PAYX_Paychex_zscore_60d", "EWS_Singapore_ret_5d", "VOD_Vodafone_zscore_60d", "ORCL_zscore_60d", "US1Y_Rate_ret_5d", "EWH_HongKong_ret_5d", "XOM_ret_20d", "EXC_Exelon_zscore_60d", "CI_Cigna_vol_20d", "ES_Evergy_ret_1d", "PAYX_Paychex_ret_20d", "BA_ret_1d", "T10Y2Y_Spread_ret_5d", "spx_momentum_3d", "HD_zscore_60d", "LMT_LockheedMartin_vol_20d", "Retail_Sales_zscore_60d", "EWJ_Japan_vol_20d", "3M_vol_20d", "PAYX_Paychex_vol_20d", "CPB_CampbellSoup_vol_20d", "US7Y_Rate_ret_20d", "BDX_Becton_Dickinson_ret_20d", "EFFR_vol_20d", "IWM_SmallCap_vol_20d", "DIS_vol_20d"], "is_new": true}, {"model_id": "new_h7_NORMAL_LogisticRegression_N30_t5", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 7, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "vix_mean_abs_ret_5d", "EWL_Switzerland_zscore_60d", "US30Y_Rate_ret_20d", "PAYX_Paychex_vol_20d", "EWM_Malaysia_zscore_60d", "EWC_Canada_zscore_60d", "HangSeng_HK_ret_1d", "Brent_Oil_FRED_ret_20d", "spx_momentum_3d", "EFFR_ret_1d", "AMD_ret_1d", "LMT_LockheedMartin_vol_20d", "AVB_AvalonBay_zscore_60d", "EXC_Exelon_ret_1d", "MSTR_Bitcoin3_ret_20d", "LUV_SouthwestAir_ret_5d", "PAYX_Paychex_ret_20d", "TXN_vol_20d", "GE_ret_1d", "MSTR_Bitcoin3_ret_5d", "MS_MorganStanley_ret_5d", "MSTR_Bitcoin3_ret_1d", "DHR_vol_20d", "ORCL_zscore_60d", "CPB_CampbellSoup_vol_20d", "EWG_Germany_vol_20d", "BTI_BritishAmerican_ret_20d", "DE_Deere_vol_20d"], "is_new": true}, {"model_id": "new_h7_NORMAL_LogisticRegression_N30_t6", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 7, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWM_Malaysia_vol_20d", "EQR_Equity_ret_1d", "MS_MorganStanley_zscore_60d", "LLY_zscore_60d", "EWS_Singapore_ret_5d", "PAYX_Paychex_vol_20d", "EWQ_France_ret_20d", "EWM_Malaysia_ret_1d", "US3M_Rate_vol_20d", "PAYX_Paychex_zscore_60d", "TGT_Target_zscore_60d", "MO_AltriaMG_ret_1d", "BA_ret_1d", "EWY_Korea_ret_20d", "ORCL_zscore_60d", "LOW_Lowes_ret_5d", "vix_acceleration_1d", "EWQ_France_zscore_60d", "spx_abs_ret_max_5d", "LMT_LockheedMartin_ret_1d", "MRK_Merck_zscore_60d", "FedFunds_zscore_60d", "ES_Evergy_ret_1d", "AVB_AvalonBay_zscore_60d", "SO_SouthernCo_ret_5d", "MSTR_Bitcoin3_ret_20d", "PG_ret_20d", "IWM_SmallCap_vol_20d"], "is_new": true}, {"model_id": "new_h7_NORMAL_LogisticRegression_N30_t7", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 7, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "heston_var_ev_h3", "HangSeng_HK_ret_5d", "vix_acceleration_1d", "EWL_Switzerland_vol_20d", "Nikkei_Japan_zscore_60d", "MO_AltriaMG_ret_1d", "US3M_Rate_vol_20d", "CTAS_Cintas_vol_20d", "IWM_SmallCap_vol_20d", "EWL_Switzerland_zscore_60d", "EFFR_vol_20d", "SJM_JM_Smucker_ret_1d", "NVDA_vol_20d", "EWG_Germany_vol_20d", "XLV_Health_zscore_60d", "ITT_ITTInc_ret_5d", "ORCL_vol_20d", "SPY_zscore_60d", "LMT_LockheedMartin_ret_1d", "Brent_Oil_FRED_ret_5d", "GE_ret_1d", "DE_Deere_vol_20d", "EWJ_Japan_vol_20d", "SBUX_ret_5d", "BA_ret_1d", "GILD_Gilead_ret_20d", "SJM_JM_Smucker_ret_5d", "Industrial_Production_zscore_60d"], "is_new": true}, {"model_id": "new_h7_STRESS_XGBoost_N5_t0", "algo": "XGBoost", "regime": "STRESS", "horizon": 7, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWL_Switzerland_zscore_60d", "TED_Spread_zscore_60d", "EXC_Exelon_ret_1d"], "is_new": true}, {"model_id": "new_h7_STRESS_XGBoost_N5_t1", "algo": "XGBoost", "regime": "STRESS", "horizon": 7, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "Brent_Oil_FRED_ret_5d", "EWA_Australia_zscore_60d", "LOW_Lowes_ret_5d"], "is_new": true}, {"model_id": "new_h7_STRESS_XGBoost_N5_t2", "algo": "XGBoost", "regime": "STRESS", "horizon": 7, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "PLD_Prologis_ret_5d", "spx_momentum_3d", "INTC_ret_5d"], "is_new": true}, {"model_id": "new_h7_STRESS_XGBoost_N5_t3", "algo": "XGBoost", "regime": "STRESS", "horizon": 7, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "vix_mean_abs_ret_5d", "XLV_Health_zscore_60d", "XOM_ret_1d"], "is_new": true}, {"model_id": "new_h7_STRESS_XGBoost_N5_t4", "algo": "XGBoost", "regime": "STRESS", "horizon": 7, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "Core_CPI_zscore_60d", "Brent_Oil_FRED_ret_20d", "TM_Telephone_ret_1d"], "is_new": true}, {"model_id": "new_h7_STRESS_XGBoost_N5_t5", "algo": "XGBoost", "regime": "STRESS", "horizon": 7, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EOG_EOGResources_ret_5d", "TM_Telephone_vol_20d", "IYR_US_REIT2_zscore_60d"], "is_new": true}, {"model_id": "new_h7_STRESS_XGBoost_N5_t6", "algo": "XGBoost", "regime": "STRESS", "horizon": 7, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "CPB_CampbellSoup_ret_5d", "AMD_ret_1d", "ORCL_zscore_60d"], "is_new": true}, {"model_id": "new_h7_STRESS_XGBoost_N5_t7", "algo": "XGBoost", "regime": "STRESS", "horizon": 7, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "SCHW_Schwab_ret_5d", "VOD_Vodafone_zscore_60d", "US1Y_Rate_ret_20d"], "is_new": true}, {"model_id": "new_h7_STRESS_XGBoost_N8_t0", "algo": "XGBoost", "regime": "STRESS", "horizon": 7, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "NEE_NextEra_ret_20d", "EWS_Singapore_ret_5d", "GILD_Gilead_ret_20d", "PPL_PPL_ret_1d", "gjr_condvar_h1", "heston_var_ev_h3"], "is_new": true}, {"model_id": "new_h7_STRESS_XGBoost_N8_t1", "algo": "XGBoost", "regime": "STRESS", "horizon": 7, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "BLK_BlackRock_zscore_60d", "ASX_Australia_vol_20d", "US3Y_Rate_ret_5d", "EWH_HongKong_ret_5d", "NVDA_vol_20d", "EWL_Switzerland_vol_20d"], "is_new": true}, {"model_id": "new_h7_STRESS_XGBoost_N8_t2", "algo": "XGBoost", "regime": "STRESS", "horizon": 7, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "BA_ret_1d", "EFFR_ret_1d", "DIS_vol_20d", "PAYX_Paychex_zscore_60d", "GE_ret_1d", "vix_acceleration_1d"], "is_new": true}, {"model_id": "new_h7_STRESS_XGBoost_N8_t3", "algo": "XGBoost", "regime": "STRESS", "horizon": 7, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "PLD_Prologis_ret_5d", "PAYX_Paychex_ret_20d", "Brent_Oil_FRED_ret_20d", "BDX_Becton_Dickinson_ret_20d", "LOW_Lowes_ret_20d", "AMZN_ret_5d"], "is_new": true}, {"model_id": "new_h7_STRESS_XGBoost_N8_t4", "algo": "XGBoost", "regime": "STRESS", "horizon": 7, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "LMT_LockheedMartin_vol_20d", "INTC_ret_1d", "CMCSA_ret_1d", "DOW_Price_zscore_60d", "CCI_CrownCastle_vol_20d", "EWG_Germany_vol_20d"], "is_new": true}, {"model_id": "new_h7_STRESS_XGBoost_N8_t5", "algo": "XGBoost", "regime": "STRESS", "horizon": 7, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWM_Malaysia_ret_1d", "XOM_ret_1d", "EWA_Australia_zscore_60d", "EWJ_Japan_vol_20d", "HD_ret_5d", "Nikkei_Japan_zscore_60d"], "is_new": true}, {"model_id": "new_h7_STRESS_XGBoost_N8_t6", "algo": "XGBoost", "regime": "STRESS", "horizon": 7, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "DHR_vol_20d", "EWY_Korea_zscore_60d", "AMD_ret_1d", "ASX_Australia_ret_5d", "CPB_CampbellSoup_vol_20d", "SBUX_ret_5d"], "is_new": true}, {"model_id": "new_h7_STRESS_XGBoost_N8_t7", "algo": "XGBoost", "regime": "STRESS", "horizon": 7, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "HD_ret_20d", "FedFunds_zscore_60d", "Retail_Sales_zscore_60d", "EWQ_France_zscore_60d", "T_ret_1d", "EWG_Germany_ret_20d"], "is_new": true}, {"model_id": "new_h7_STRESS_XGBoost_N10_t0", "algo": "XGBoost", "regime": "STRESS", "horizon": 7, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "AVB_AvalonBay_zscore_60d", "PAYX_Paychex_vol_20d", "Nikkei_Japan_vol_20d", "EMR_Emerson_ret_20d", "SPY_zscore_60d", "AMGN_Amgen_ret_1d", "VVIX_ret_20d", "US6M_Rate_ret_20d"], "is_new": true}, {"model_id": "new_h7_STRESS_XGBoost_N10_t1", "algo": "XGBoost", "regime": "STRESS", "horizon": 7, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "DHR_ret_1d", "IBEX_Spain_ret_20d", "heston_ev_h3", "AMD_ret_1d", "US30Y_Rate_ret_20d", "gjr_condvar_h1", "HUM_Humana_ret_5d", "SJM_JM_Smucker_ret_1d"], "is_new": true}, {"model_id": "new_h7_STRESS_XGBoost_N10_t2", "algo": "XGBoost", "regime": "STRESS", "horizon": 7, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "DHR_vol_20d", "PAYX_Paychex_ret_20d", "JNJ_ret_1d", "SCHW_Schwab_ret_5d", "US1Y_Rate_ret_5d", "XLB_Materials_zscore_60d", "TM_Telephone_ret_1d", "NFCI_ret_5d"], "is_new": true}, {"model_id": "new_h7_STRESS_XGBoost_N10_t3", "algo": "XGBoost", "regime": "STRESS", "horizon": 7, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "Core_PCE_zscore_60d", "ORCL_vol_20d", "HD_ret_1d", "XLY_Disc_vol_20d", "AXP_Amex_vol_20d", "DAX_Germany_zscore_60d", "HangSeng_HK_vol_20d", "TM_Telephone_vol_20d"], "is_new": true}, {"model_id": "new_h7_STRESS_XGBoost_N10_t4", "algo": "XGBoost", "regime": "STRESS", "horizon": 7, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWA_Australia_zscore_60d", "heston_var_ev_h5", "MS_MorganStanley_zscore_60d", "spx_abs_ret_max_5d", "PPL_PPL_ret_1d", "M_Macys_vol_20d", "EWL_Switzerland_vol_20d", "AXP_Amex_vol_20d"], "is_new": true}, {"model_id": "new_h7_STRESS_XGBoost_N10_t5", "algo": "XGBoost", "regime": "STRESS", "horizon": 7, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "PLD_Prologis_ret_5d", "LMT_LockheedMartin_vol_20d", "PCAR_PaccarInc_ret_5d", "heston_ev_h3", "EWL_Switzerland_vol_20d", "CPB_CampbellSoup_zscore_60d", "US1Y_Rate_ret_5d", "EQIX_Equinix_ret_5d"], "is_new": true}, {"model_id": "new_h7_STRESS_XGBoost_N10_t6", "algo": "XGBoost", "regime": "STRESS", "horizon": 7, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWM_Malaysia_ret_1d", "EWY_Korea_ret_20d", "US1Y_Rate_ret_5d", "MSTR_Bitcoin3_ret_20d", "BLK_BlackRock_zscore_60d", "PAYX_Paychex_vol_20d", "US3M_Rate_zscore_60d", "EFFR_vol_20d"], "is_new": true}, {"model_id": "new_h7_STRESS_XGBoost_N10_t7", "algo": "XGBoost", "regime": "STRESS", "horizon": 7, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EFFR_vol_20d", "ENB_EnbridgeInc_ret_1d", "DE_Deere_vol_20d", "IWM_SmallCap_vol_20d", "GE_ret_1d", "HangSeng_HK_ret_5d", "MSTR_Bitcoin3_ret_1d", "hmm_p_stress"], "is_new": true}, {"model_id": "new_h7_STRESS_XGBoost_N12_t0", "algo": "XGBoost", "regime": "STRESS", "horizon": 7, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "MSTR_Bitcoin3_ret_20d", "NFCI_ret_5d", "US3M_Rate_vol_20d", "CPB_CampbellSoup_vol_20d", "WTI_Oil_FRED_zscore_60d", "SBUX_ret_5d", "MS_MorganStanley_ret_1d", "US3M_Rate_zscore_60d", "MS_MorganStanley_ret_5d", "EWC_Canada_zscore_60d"], "is_new": true}, {"model_id": "new_h7_STRESS_XGBoost_N12_t1", "algo": "XGBoost", "regime": "STRESS", "horizon": 7, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "CMCSA_ret_1d", "INTC_ret_1d", "XLY_Disc_vol_20d", "US30Y_Rate_ret_20d", "EXC_Exelon_zscore_60d", "GE_ret_1d", "VOD_Vodafone_zscore_60d", "Nikkei_Japan_zscore_60d", "PFE_ret_1d", "EXC_Exelon_ret_1d"], "is_new": true}, {"model_id": "new_h7_STRESS_XGBoost_N12_t2", "algo": "XGBoost", "regime": "STRESS", "horizon": 7, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "M_Macys_vol_20d", "SJM_JM_Smucker_ret_1d", "MRK_Merck_zscore_60d", "MS_MorganStanley_ret_1d", "US3Y_Rate_ret_5d", "vix_acceleration_1d", "HD_ret_5d", "SLB_Schlumberger_ret_5d", "LOW_Lowes_ret_5d", "heston_var_ev_h3"], "is_new": true}, {"model_id": "new_h7_STRESS_XGBoost_N12_t3", "algo": "XGBoost", "regime": "STRESS", "horizon": 7, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "VVIX_ret_20d", "SLB_Schlumberger_ret_1d", "SJM_JM_Smucker_ret_5d", "WTI_Oil_FRED_zscore_60d", "BA_ret_1d", "T_ret_1d", "CCI_CrownCastle_vol_20d", "IWM_SmallCap_vol_20d", "INTC_ret_5d", "vix_acceleration_1d"], "is_new": true}, {"model_id": "new_h7_STRESS_XGBoost_N12_t4", "algo": "XGBoost", "regime": "STRESS", "horizon": 7, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWL_Switzerland_vol_20d", "spx_vol_5d", "LMT_LockheedMartin_vol_20d", "hmm_p_stress", "MSTR_Bitcoin3_ret_5d", "EMR_Emerson_ret_20d", "Nikkei_Japan_vol_20d", "HD_ret_20d", "heston_var_ev_h5", "AXP_Amex_vol_20d"], "is_new": true}, {"model_id": "new_h7_STRESS_XGBoost_N12_t5", "algo": "XGBoost", "regime": "STRESS", "horizon": 7, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "CMCSA_ret_1d", "HangSeng_HK_ret_1d", "WTI_Oil_FRED_zscore_60d", "LUV_SouthwestAir_ret_5d", "PCAR_PaccarInc_ret_5d", "DHR_ret_1d", "heston_var_ev_h3", "EFFR_vol_20d", "US30Y_Rate_ret_20d", "CPB_CampbellSoup_ret_5d"], "is_new": true}, {"model_id": "new_h7_STRESS_XGBoost_N12_t6", "algo": "XGBoost", "regime": "STRESS", "horizon": 7, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "AXP_Amex_ret_20d", "IWM_SmallCap_vol_20d", "DHR_ret_1d", "MSTR_Bitcoin3_ret_5d", "PAYX_Paychex_vol_20d", "CCI_CrownCastle_vol_20d", "NOC_Northrop_ret_20d", "VRP_ma5", "PFE_ret_1d", "US1Y_Rate_ret_20d"], "is_new": true}, {"model_id": "new_h7_STRESS_XGBoost_N12_t7", "algo": "XGBoost", "regime": "STRESS", "horizon": 7, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "PCAR_PaccarInc_ret_5d", "EWL_Switzerland_zscore_60d", "DAX_Germany_vol_20d", "HangSeng_HK_ret_5d", "T_ret_1d", "CMCSA_ret_1d", "EFFR_ret_1d", "FedFunds_zscore_60d", "SBUX_zscore_60d", "VRP_ma5"], "is_new": true}, {"model_id": "new_h7_STRESS_XGBoost_N15_t0", "algo": "XGBoost", "regime": "STRESS", "horizon": 7, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "vix_mean_abs_ret_5d", "AMGN_Amgen_ret_1d", "EFFR_ret_1d", "AXP_Amex_ret_20d", "MS_MorganStanley_zscore_60d", "Industrial_Production_zscore_60d", "BTI_BritishAmerican_ret_20d", "INTC_ret_1d", "PAYX_Paychex_zscore_60d", "DHR_ret_1d", "AMZN_ret_5d", "EQR_Equity_ret_1d", "EWY_Korea_zscore_60d"], "is_new": true}, {"model_id": "new_h7_STRESS_XGBoost_N15_t1", "algo": "XGBoost", "regime": "STRESS", "horizon": 7, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWJ_Japan_vol_20d", "MSTR_Bitcoin3_ret_1d", "DOW_Price_zscore_60d", "Nikkei_Japan_zscore_60d", "EWG_Germany_ret_20d", "VRP_ma5", "MS_MorganStanley_ret_1d", "EWY_Korea_ret_20d", "US1Y_Rate_ret_5d", "IWM_SmallCap_vol_20d", "WTI_Oil_FRED_zscore_60d", "AXP_Amex_ret_20d", "LOW_Lowes_ret_5d"], "is_new": true}, {"model_id": "new_h7_STRESS_XGBoost_N15_t2", "algo": "XGBoost", "regime": "STRESS", "horizon": 7, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "LLY_zscore_60d", "NFCI_ret_5d", "MS_MorganStanley_zscore_60d", "MO_AltriaMG_ret_1d", "HD_ret_20d", "T10Y2Y_Spread_ret_5d", "CPB_CampbellSoup_vol_20d", "VOD_Vodafone_zscore_60d", "BA_ret_1d", "CPB_CampbellSoup_zscore_60d", "heston_var_ev_h7", "Retail_Sales_zscore_60d", "AXP_Amex_vol_20d"], "is_new": true}, {"model_id": "new_h7_STRESS_XGBoost_N15_t3", "algo": "XGBoost", "regime": "STRESS", "horizon": 7, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "Retail_Sales_zscore_60d", "NVDA_vol_20d", "GE_ret_1d", "EWQ_France_zscore_60d", "spx_vol_5d", "VOD_Vodafone_zscore_60d", "VRP_ma5", "ORCL_vol_20d", "ASX_Australia_ret_5d", "MS_MorganStanley_zscore_60d", "PLD_Prologis_ret_5d", "LOW_Lowes_ret_20d", "DHR_ret_1d"], "is_new": true}, {"model_id": "new_h7_STRESS_XGBoost_N15_t4", "algo": "XGBoost", "regime": "STRESS", "horizon": 7, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "SBUX_ret_5d", "hmm_p_stress", "BTI_BritishAmerican_ret_5d", "EOG_EOGResources_vol_20d", "XLV_Health_zscore_60d", "PAYX_Paychex_vol_20d", "spx_momentum_3d", "ENB_EnbridgeInc_ret_1d", "LMT_LockheedMartin_vol_20d", "TXN_vol_20d", "QQQ_vol_20d", "CPB_CampbellSoup_zscore_60d", "EWS_Singapore_ret_5d"], "is_new": true}, {"model_id": "new_h7_STRESS_XGBoost_N15_t5", "algo": "XGBoost", "regime": "STRESS", "horizon": 7, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWY_Korea_zscore_60d", "DOW_Price_zscore_60d", "hmm_p_stress", "SPY_zscore_60d", "US6M_Rate_ret_20d", "AMZN_ret_5d", "AMT_AmericanTower_ret_1d", "MO_AltriaMG_ret_1d", "MS_MorganStanley_zscore_60d", "PLD_Prologis_ret_5d", "HangSeng_HK_ret_5d", "BTI_BritishAmerican_ret_5d", "DHR_vol_20d"], "is_new": true}, {"model_id": "new_h7_STRESS_XGBoost_N15_t6", "algo": "XGBoost", "regime": "STRESS", "horizon": 7, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "IWM_SmallCap_vol_20d", "ASX_Australia_vol_20d", "HD_ret_1d", "GE_ret_1d", "TM_Telephone_vol_20d", "EWG_Germany_vol_20d", "US3M_Rate_vol_20d", "BTI_BritishAmerican_ret_20d", "PPL_PPL_ret_1d", "EWA_Australia_ret_1d", "AXP_Amex_ret_20d", "AMT_AmericanTower_ret_1d", "US30Y_Rate_ret_20d"], "is_new": true}, {"model_id": "new_h7_STRESS_XGBoost_N15_t7", "algo": "XGBoost", "regime": "STRESS", "horizon": 7, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "US5Y_Rate_ret_5d", "SBUX_zscore_60d", "BA_ret_1d", "HD_ret_1d", "CPB_CampbellSoup_ret_20d", "INTC_ret_1d", "QQQ_vol_20d", "EWY_Korea_zscore_60d", "DE_Deere_ret_5d", "XOM_ret_1d", "FedFunds_zscore_60d", "EWG_Germany_vol_20d", "DIS_vol_20d"], "is_new": true}, {"model_id": "new_h7_STRESS_XGBoost_N20_t0", "algo": "XGBoost", "regime": "STRESS", "horizon": 7, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "heston_var_ev_h3", "TM_Telephone_ret_1d", "DIS_vol_20d", "US3M_Rate_zscore_60d", "EQR_Equity_ret_1d", "SBUX_ret_5d", "EWJ_Japan_vol_20d", "BA_ret_1d", "gjr_condvar_h1", "EWY_Korea_ret_20d", "XLY_Disc_vol_20d", "EWQ_France_zscore_60d", "PAYX_Paychex_vol_20d", "ASX_Australia_ret_5d", "EWH_HongKong_ret_5d", "HD_ret_20d", "US30Y_Rate_ret_20d", "ITT_ITTInc_ret_5d"], "is_new": true}, {"model_id": "new_h7_STRESS_XGBoost_N20_t1", "algo": "XGBoost", "regime": "STRESS", "horizon": 7, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "hmm_p_stress", "US3M_Rate_zscore_60d", "VVIX_ret_20d", "3M_ret_5d", "heston_var_ev_h3", "BDX_Becton_Dickinson_ret_20d", "vix_mean_abs_ret_5d", "VRP_ma5", "Retail_Sales_zscore_60d", "EXC_Exelon_ret_1d", "EWL_Switzerland_zscore_60d", "ORCL_zscore_60d", "LLY_zscore_60d", "PAYX_Paychex_vol_20d", "US3M_Rate_vol_20d", "EFFR_ret_1d", "SBUX_zscore_60d", "SLB_Schlumberger_ret_1d"], "is_new": true}, {"model_id": "new_h7_STRESS_XGBoost_N20_t2", "algo": "XGBoost", "regime": "STRESS", "horizon": 7, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "MSTR_Bitcoin3_ret_20d", "MS_MorganStanley_ret_5d", "LMT_LockheedMartin_ret_1d", "BDX_Becton_Dickinson_ret_20d", "HD_zscore_60d", "EWM_Malaysia_vol_20d", "HangSeng_HK_ret_1d", "GE_ret_1d", "TM_Telephone_vol_20d", "M_Macys_vol_20d", "ORCL_vol_20d", "CTAS_Cintas_vol_20d", "heston_ev_h3", "CPB_CampbellSoup_zscore_60d", "PAYX_Paychex_ret_20d", "PG_ret_20d", "CLX_Clorox_vol_20d", "NEE_NextEra_ret_20d"], "is_new": true}, {"model_id": "new_h7_STRESS_XGBoost_N20_t3", "algo": "XGBoost", "regime": "STRESS", "horizon": 7, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "HD_ret_1d", "SLB_Schlumberger_ret_5d", "EFFR_ret_1d", "US3Y_Rate_ret_5d", "SJM_JM_Smucker_ret_5d", "CMCSA_ret_1d", "EWJ_Japan_vol_20d", "EQR_Equity_ret_1d", "AXP_Amex_ret_20d", "DAX_Germany_zscore_60d", "SCHW_Schwab_ret_5d", "DIS_vol_20d", "LOW_Lowes_ret_20d", "Retail_Sales_zscore_60d", "Industrial_Production_zscore_60d", "ENB_EnbridgeInc_ret_1d", "NEE_NextEra_ret_20d", "EQIX_Equinix_ret_5d"], "is_new": true}, {"model_id": "new_h7_STRESS_XGBoost_N20_t4", "algo": "XGBoost", "regime": "STRESS", "horizon": 7, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "Core_PCE_zscore_60d", "INTC_ret_5d", "EOG_EOGResources_vol_20d", "DHR_ret_1d", "US3M_Rate_zscore_60d", "EWL_Switzerland_vol_20d", "ASX_Australia_vol_20d", "CTAS_Cintas_vol_20d", "CPB_CampbellSoup_ret_20d", "SBUX_zscore_60d", "WTI_Oil_FRED_zscore_60d", "MSTR_Bitcoin3_ret_5d", "TM_Telephone_ret_1d", "LOW_Lowes_ret_5d", "HangSeng_HK_ret_1d", "LMT_LockheedMartin_ret_1d", "BLK_BlackRock_zscore_60d", "T_ret_1d"], "is_new": true}, {"model_id": "new_h7_STRESS_XGBoost_N20_t5", "algo": "XGBoost", "regime": "STRESS", "horizon": 7, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "HD_zscore_60d", "T_ret_1d", "Nikkei_Japan_zscore_60d", "US30Y_Rate_ret_20d", "BA_ret_1d", "BDX_Becton_Dickinson_ret_20d", "CPB_CampbellSoup_zscore_60d", "Industrial_Production_zscore_60d", "M_Macys_vol_20d", "EWQ_France_ret_20d", "Core_CPI_zscore_60d", "GD_GeneralDynamics_zscore_60d", "EWJ_Japan_vol_20d", "PG_ret_20d", "AMGN_Amgen_ret_1d", "vix_mean_abs_ret_5d", "CPB_CampbellSoup_vol_20d", "HangSeng_HK_vol_20d"], "is_new": true}, {"model_id": "new_h7_STRESS_XGBoost_N20_t6", "algo": "XGBoost", "regime": "STRESS", "horizon": 7, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "HUM_Humana_ret_5d", "Michigan_Sentiment_ret_20d", "EWG_Germany_ret_20d", "INTC_ret_5d", "TM_Telephone_ret_1d", "IYR_US_REIT2_zscore_60d", "DAX_Germany_zscore_60d", "CMCSA_ret_1d", "CLX_Clorox_vol_20d", "BA_ret_1d", "DAX_Germany_vol_20d", "SCHW_Schwab_ret_5d", "WTI_Oil_FRED_zscore_60d", "VOD_Vodafone_zscore_60d", "EOG_EOGResources_ret_5d", "BDX_Becton_Dickinson_ret_20d", "PFE_ret_1d", "Retail_Sales_zscore_60d"], "is_new": true}, {"model_id": "new_h7_STRESS_XGBoost_N20_t7", "algo": "XGBoost", "regime": "STRESS", "horizon": 7, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "DIS_vol_20d", "XOM_ret_1d", "PPL_PPL_ret_1d", "MRK_Merck_zscore_60d", "EWA_Australia_zscore_60d", "EWL_Switzerland_zscore_60d", "EWL_Switzerland_vol_20d", "heston_var_ev_h5", "IYM_BasicMaterials_ret_20d", "NVDA_vol_20d", "LUV_SouthwestAir_ret_5d", "PAYX_Paychex_zscore_60d", "EFFR_ret_1d", "VOD_Vodafone_zscore_60d", "HangSeng_HK_vol_20d", "MS_MorganStanley_ret_1d", "AORD_AUS_zscore_60d", "DAX_Germany_vol_20d"], "is_new": true}, {"model_id": "new_h7_STRESS_XGBoost_N25_t0", "algo": "XGBoost", "regime": "STRESS", "horizon": 7, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "VRP_ma5", "MS_MorganStanley_ret_1d", "US6M_Rate_ret_20d", "heston_ev_h3", "EWM_Malaysia_zscore_60d", "EWM_Malaysia_ret_1d", "SJM_JM_Smucker_ret_5d", "XLV_Health_zscore_60d", "FedFunds_zscore_60d", "SJM_JM_Smucker_ret_1d", "EQR_Equity_ret_1d", "Nikkei_Japan_vol_20d", "VOD_Vodafone_zscore_60d", "Retail_Sales_zscore_60d", "AXP_Amex_vol_20d", "QQQ_vol_20d", "gjr_condvar_h1", "PFE_ret_1d", "AMD_ret_1d", "TXN_vol_20d", "SBUX_vol_20d", "EWQ_France_zscore_60d", "EWG_Germany_vol_20d"], "is_new": true}, {"model_id": "new_h7_STRESS_XGBoost_N25_t1", "algo": "XGBoost", "regime": "STRESS", "horizon": 7, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "XLF_Fin_vol_20d", "Retail_Sales_zscore_60d", "LOW_Lowes_ret_20d", "BLK_BlackRock_zscore_60d", "PAYX_Paychex_vol_20d", "US30Y_Rate_ret_20d", "EWM_Malaysia_vol_20d", "SO_SouthernCo_ret_5d", "TED_Spread_vol_20d", "NVDA_vol_20d", "LLY_zscore_60d", "LMT_LockheedMartin_ret_1d", "hmm_p_stress", "vix_mean_abs_ret_5d", "Core_CPI_zscore_60d", "EWG_Germany_vol_20d", "IWM_SmallCap_vol_20d", "HangSeng_HK_ret_1d", "MS_MorganStanley_zscore_60d", "Michigan_Sentiment_ret_20d", "gjr_condvar_h1", "DIS_vol_20d", "DHR_ret_1d"], "is_new": true}, {"model_id": "new_h7_STRESS_XGBoost_N25_t2", "algo": "XGBoost", "regime": "STRESS", "horizon": 7, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWY_Korea_ret_20d", "XLK_Tech_zscore_60d", "Michigan_Sentiment_ret_20d", "US30Y_Rate_ret_20d", "US1Y_Rate_ret_5d", "NWL_Newell_ret_20d", "BLK_BlackRock_zscore_60d", "BTI_BritishAmerican_ret_5d", "AXP_Amex_vol_20d", "EMR_Emerson_ret_20d", "Industrial_Production_zscore_60d", "MS_MorganStanley_zscore_60d", "VRP_ma5", "TED_Spread_vol_20d", "ENB_EnbridgeInc_ret_1d", "VOD_Vodafone_zscore_60d", "ORCL_zscore_60d", "hmm_p_stress", "US3Y_Rate_ret_5d", "PAYX_Paychex_ret_20d", "CLX_Clorox_vol_20d", "HD_ret_20d", "ORCL_vol_20d"], "is_new": true}, {"model_id": "new_h7_STRESS_XGBoost_N25_t3", "algo": "XGBoost", "regime": "STRESS", "horizon": 7, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "PLD_Prologis_ret_5d", "PG_ret_20d", "Industrial_Production_zscore_60d", "GE_ret_1d", "AMGN_Amgen_ret_1d", "IYM_BasicMaterials_ret_20d", "MO_AltriaMG_ret_1d", "LOW_Lowes_ret_20d", "AVB_AvalonBay_zscore_60d", "TXN_vol_20d", "EWA_Australia_zscore_60d", "FedFunds_zscore_60d", "T_ret_1d", "GILD_Gilead_ret_20d", "3M_ret_5d", "EWH_HongKong_ret_5d", "MS_MorganStanley_zscore_60d", "DAX_Germany_zscore_60d", "US1Y_Rate_ret_20d", "IWM_SmallCap_vol_20d", "ITT_ITTInc_ret_5d", "HD_ret_20d", "IBEX_Spain_ret_20d"], "is_new": true}, {"model_id": "new_h7_STRESS_XGBoost_N25_t4", "algo": "XGBoost", "regime": "STRESS", "horizon": 7, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "3M_vol_20d", "US5Y_Rate_ret_5d", "ENB_EnbridgeInc_ret_1d", "Michigan_Sentiment_ret_20d", "T10Y2Y_Spread_ret_5d", "US3M_Rate_zscore_60d", "TED_Spread_zscore_60d", "DAX_Germany_vol_20d", "DHR_vol_20d", "CMCSA_ret_1d", "CPB_CampbellSoup_ret_5d", "TM_Telephone_vol_20d", "INTC_ret_5d", "US30Y_Rate_ret_20d", "TGT_Target_zscore_60d", "IYR_US_REIT2_zscore_60d", "EWL_Switzerland_zscore_60d", "EWG_Germany_vol_20d", "EMR_Emerson_ret_20d", "VVIX_ret_20d", "MO_AltriaMG_ret_1d", "ORCL_zscore_60d", "XOM_ret_20d"], "is_new": true}, {"model_id": "new_h7_STRESS_XGBoost_N25_t5", "algo": "XGBoost", "regime": "STRESS", "horizon": 7, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "WTI_Oil_FRED_zscore_60d", "LLY_zscore_60d", "spx_vol_5d", "heston_var_ev_h7", "DE_Deere_ret_5d", "INTC_ret_5d", "MS_MorganStanley_ret_5d", "EWJ_Japan_vol_20d", "BLK_BlackRock_zscore_60d", "TED_Spread_zscore_60d", "3M_vol_20d", "AXP_Amex_vol_20d", "ASX_Australia_vol_20d", "XOM_ret_1d", "EWQ_France_ret_20d", "T_ret_1d", "CTAS_Cintas_vol_20d", "AMZN_ret_5d", "PAYX_Paychex_zscore_60d", "GE_ret_1d", "XLF_Fin_vol_20d", "EOG_EOGResources_vol_20d", "Nikkei_Japan_zscore_60d"], "is_new": true}, {"model_id": "new_h7_STRESS_XGBoost_N25_t6", "algo": "XGBoost", "regime": "STRESS", "horizon": 7, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "BA_ret_1d", "EWM_Malaysia_ret_1d", "Brent_Oil_FRED_ret_20d", "DHR_ret_1d", "TED_Spread_vol_20d", "AXP_Amex_ret_20d", "PAYX_Paychex_vol_20d", "XOM_ret_20d", "DHR_vol_20d", "VVIX_ret_20d", "XLK_Tech_zscore_60d", "HangSeng_HK_ret_5d", "HD_ret_20d", "PAYX_Paychex_zscore_60d", "EXC_Exelon_ret_1d", "MSTR_Bitcoin3_ret_20d", "EMR_Emerson_ret_20d", "EQIX_Equinix_ret_5d", "Michigan_Sentiment_ret_20d", "MS_MorganStanley_ret_1d", "XOM_ret_1d", "ASX_Australia_vol_20d", "SJM_JM_Smucker_ret_1d"], "is_new": true}, {"model_id": "new_h7_STRESS_XGBoost_N25_t7", "algo": "XGBoost", "regime": "STRESS", "horizon": 7, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "gjr_condvar_h1", "CLX_Clorox_vol_20d", "SLB_Schlumberger_ret_5d", "M_Macys_vol_20d", "EWM_Malaysia_vol_20d", "EXC_Exelon_ret_1d", "IYR_US_REIT2_zscore_60d", "LUV_SouthwestAir_ret_5d", "INTC_ret_1d", "HangSeng_HK_ret_5d", "PPL_PPL_ret_1d", "AMD_ret_5d", "NOC_Northrop_ret_20d", "ASX_Australia_ret_5d", "SCHW_Schwab_ret_5d", "SJM_JM_Smucker_ret_5d", "BA_ret_1d", "heston_var_ev_h5", "IBEX_Spain_ret_20d", "3M_vol_20d", "PAYX_Paychex_ret_20d", "CTAS_Cintas_vol_20d", "EOG_EOGResources_ret_5d"], "is_new": true}, {"model_id": "new_h7_STRESS_XGBoost_N30_t0", "algo": "XGBoost", "regime": "STRESS", "horizon": 7, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "CCI_CrownCastle_vol_20d", "SBUX_zscore_60d", "Nikkei_Japan_vol_20d", "DHR_vol_20d", "vix_mean_abs_ret_5d", "IYR_US_REIT2_zscore_60d", "SJM_JM_Smucker_ret_5d", "SO_SouthernCo_ret_5d", "PPL_PPL_ret_1d", "EWM_Malaysia_zscore_60d", "CI_Cigna_vol_20d", "GILD_Gilead_ret_20d", "ITT_ITTInc_ret_5d", "AVB_AvalonBay_zscore_60d", "heston_ev_h3", "EWY_Korea_ret_20d", "CPB_CampbellSoup_zscore_60d", "MO_AltriaMG_ret_1d", "DE_Deere_vol_20d", "EWH_HongKong_ret_5d", "LUV_SouthwestAir_ret_5d", "EOG_EOGResources_ret_5d", "XOM_ret_1d", "EWC_Canada_zscore_60d", "EQR_Equity_ret_1d", "DIS_vol_20d", "ASX_Australia_ret_5d", "US1Y_Rate_ret_20d"], "is_new": true}, {"model_id": "new_h7_STRESS_XGBoost_N30_t1", "algo": "XGBoost", "regime": "STRESS", "horizon": 7, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "IBEX_Spain_ret_20d", "XLF_Fin_vol_20d", "VVIX_ret_20d", "EWQ_France_zscore_60d", "spx_vol_5d", "CCI_CrownCastle_vol_20d", "WTI_Oil_FRED_zscore_60d", "BA_ret_1d", "3M_vol_20d", "CPB_CampbellSoup_ret_20d", "CMCSA_ret_1d", "DE_Deere_ret_5d", "SO_SouthernCo_ret_5d", "GD_GeneralDynamics_zscore_60d", "heston_var_ev_h5", "CPB_CampbellSoup_ret_5d", "SCHW_Schwab_ret_5d", "DHR_ret_1d", "EWL_Switzerland_zscore_60d", "EXC_Exelon_zscore_60d", "spx_abs_ret_max_5d", "MS_MorganStanley_ret_5d", "SJM_JM_Smucker_ret_1d", "SBUX_vol_20d", "US3Y_Rate_ret_5d", "LUV_SouthwestAir_ret_5d", "EMR_Emerson_ret_20d", "MS_MorganStanley_zscore_60d"], "is_new": true}, {"model_id": "new_h7_STRESS_XGBoost_N30_t2", "algo": "XGBoost", "regime": "STRESS", "horizon": 7, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "SLB_Schlumberger_ret_5d", "CI_Cigna_vol_20d", "INTC_ret_5d", "FedFunds_zscore_60d", "DOW_Price_zscore_60d", "EWQ_France_ret_20d", "TM_Telephone_vol_20d", "EQR_Equity_ret_1d", "US7Y_Rate_ret_20d", "CCI_CrownCastle_vol_20d", "VRP_ma5", "NOC_Northrop_ret_20d", "XLF_Fin_vol_20d", "vix_mean_abs_ret_5d", "T_ret_1d", "AVB_AvalonBay_zscore_60d", "HD_ret_1d", "US5Y_Rate_ret_5d", "Brent_Oil_FRED_ret_20d", "VVIX_ret_20d", "XLV_Health_zscore_60d", "BTI_BritishAmerican_ret_5d", "SBUX_ret_5d", "3M_ret_5d", "AMGN_Amgen_ret_1d", "Core_CPI_zscore_60d", "MRK_Merck_zscore_60d", "US3Y_Rate_ret_5d"], "is_new": true}, {"model_id": "new_h7_STRESS_XGBoost_N30_t3", "algo": "XGBoost", "regime": "STRESS", "horizon": 7, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "PAYX_Paychex_zscore_60d", "HangSeng_HK_ret_1d", "GE_ret_1d", "EWM_Malaysia_ret_1d", "gjr_condvar_h1", "EXC_Exelon_zscore_60d", "QQQ_vol_20d", "XOM_ret_20d", "SLB_Schlumberger_ret_1d", "DHR_vol_20d", "CLX_Clorox_vol_20d", "PLD_Prologis_ret_5d", "EWL_Switzerland_vol_20d", "US1Y_Rate_ret_5d", "US3M_Rate_vol_20d", "Core_CPI_zscore_60d", "HD_ret_20d", "LOW_Lowes_ret_5d", "LUV_SouthwestAir_ret_5d", "AMD_ret_1d", "LMT_LockheedMartin_vol_20d", "NFCI_ret_5d", "IBEX_Spain_ret_20d", "EFFR_vol_20d", "AMGN_Amgen_ret_1d", "DE_Deere_vol_20d", "HD_ret_1d", "GD_GeneralDynamics_zscore_60d"], "is_new": true}, {"model_id": "new_h7_STRESS_XGBoost_N30_t4", "algo": "XGBoost", "regime": "STRESS", "horizon": 7, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "US6M_Rate_ret_20d", "MO_AltriaMG_ret_1d", "DHR_vol_20d", "JNJ_ret_1d", "ENB_EnbridgeInc_ret_1d", "US5Y_Rate_ret_5d", "ES_Evergy_ret_1d", "HD_ret_5d", "DE_Deere_vol_20d", "BA_ret_1d", "US3M_Rate_vol_20d", "AMD_ret_1d", "HD_zscore_60d", "EFFR_ret_1d", "LMT_LockheedMartin_ret_1d", "TGT_Target_zscore_60d", "XOM_ret_1d", "EWM_Malaysia_zscore_60d", "EQIX_Equinix_ret_5d", "US3Y_Rate_ret_5d", "CMCSA_ret_1d", "EWA_Australia_ret_1d", "Retail_Sales_zscore_60d", "US1Y_Rate_ret_5d", "ASX_Australia_vol_20d", "US7Y_Rate_ret_20d", "SCHW_Schwab_ret_5d", "heston_var_ev_h3"], "is_new": true}, {"model_id": "new_h7_STRESS_XGBoost_N30_t5", "algo": "XGBoost", "regime": "STRESS", "horizon": 7, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "HD_ret_1d", "M_Macys_vol_20d", "EWL_Switzerland_zscore_60d", "EWA_Australia_ret_1d", "EQR_Equity_ret_1d", "PG_ret_20d", "SPY_zscore_60d", "US1Y_Rate_ret_20d", "HD_ret_20d", "ITT_ITTInc_ret_5d", "XOM_ret_1d", "spx_vol_5d", "AMZN_ret_5d", "IWM_SmallCap_vol_20d", "AMD_ret_1d", "ORCL_vol_20d", "VOD_Vodafone_zscore_60d", "heston_var_ev_h5", "SBUX_vol_20d", "MRK_Merck_zscore_60d", "TED_Spread_zscore_60d", "INTC_ret_1d", "DOW_Price_zscore_60d", "LMT_LockheedMartin_ret_1d", "SBUX_zscore_60d", "ENB_EnbridgeInc_ret_1d", "GE_ret_1d", "EFFR_ret_1d"], "is_new": true}, {"model_id": "new_h7_STRESS_XGBoost_N30_t6", "algo": "XGBoost", "regime": "STRESS", "horizon": 7, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "AMT_AmericanTower_ret_1d", "AXP_Amex_ret_20d", "MS_MorganStanley_zscore_60d", "US3Y_Rate_ret_5d", "NWL_Newell_ret_20d", "EXC_Exelon_ret_1d", "ORCL_vol_20d", "XLV_Health_zscore_60d", "AMZN_ret_5d", "Core_CPI_zscore_60d", "NEE_NextEra_ret_20d", "HUM_Humana_ret_5d", "GILD_Gilead_ret_20d", "DHR_ret_1d", "EWQ_France_ret_20d", "PAYX_Paychex_zscore_60d", "AMD_ret_1d", "CCI_CrownCastle_vol_20d", "HD_zscore_60d", "BTI_BritishAmerican_ret_5d", "Industrial_Production_zscore_60d", "LUV_SouthwestAir_ret_5d", "BTI_BritishAmerican_ret_20d", "vix_acceleration_1d", "LMT_LockheedMartin_ret_1d", "CTAS_Cintas_vol_20d", "ORCL_zscore_60d", "SBUX_vol_20d"], "is_new": true}, {"model_id": "new_h7_STRESS_XGBoost_N30_t7", "algo": "XGBoost", "regime": "STRESS", "horizon": 7, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "hmm_p_stress", "EFFR_ret_1d", "EWQ_France_ret_20d", "MRK_Merck_zscore_60d", "heston_var_ev_h5", "SJM_JM_Smucker_ret_5d", "PAYX_Paychex_zscore_60d", "MSTR_Bitcoin3_ret_1d", "MSTR_Bitcoin3_ret_20d", "EFFR_vol_20d", "MS_MorganStanley_ret_5d", "XLK_Tech_zscore_60d", "AXP_Amex_vol_20d", "US7Y_Rate_ret_20d", "CI_Cigna_vol_20d", "XLV_Health_zscore_60d", "Nikkei_Japan_zscore_60d", "EWQ_France_zscore_60d", "ENB_EnbridgeInc_ret_1d", "EWM_Malaysia_zscore_60d", "MS_MorganStanley_ret_1d", "heston_ev_h3", "spx_momentum_3d", "EWA_Australia_ret_1d", "ORCL_zscore_60d", "TED_Spread_vol_20d", "ORCL_vol_20d", "EWJ_Japan_vol_20d"], "is_new": true}, {"model_id": "new_h7_STRESS_LightGBM_N5_t0", "algo": "LightGBM", "regime": "STRESS", "horizon": 7, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "gjr_condvar_h1", "XOM_ret_20d", "EWH_HongKong_ret_5d"], "is_new": true}, {"model_id": "new_h7_STRESS_LightGBM_N5_t1", "algo": "LightGBM", "regime": "STRESS", "horizon": 7, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "TED_Spread_vol_20d", "AXP_Amex_vol_20d", "EWS_Singapore_ret_5d"], "is_new": true}, {"model_id": "new_h7_STRESS_LightGBM_N5_t2", "algo": "LightGBM", "regime": "STRESS", "horizon": 7, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "NWL_Newell_ret_20d", "HangSeng_HK_vol_20d", "US3Y_Rate_ret_5d"], "is_new": true}, {"model_id": "new_h7_STRESS_LightGBM_N5_t3", "algo": "LightGBM", "regime": "STRESS", "horizon": 7, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "SCHW_Schwab_ret_5d", "ASX_Australia_vol_20d", "ITT_ITTInc_ret_5d"], "is_new": true}, {"model_id": "new_h7_STRESS_LightGBM_N5_t4", "algo": "LightGBM", "regime": "STRESS", "horizon": 7, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "SBUX_ret_5d", "CPB_CampbellSoup_ret_20d", "HD_zscore_60d"], "is_new": true}, {"model_id": "new_h7_STRESS_LightGBM_N5_t5", "algo": "LightGBM", "regime": "STRESS", "horizon": 7, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "XLY_Disc_vol_20d", "XLV_Health_zscore_60d", "LLY_zscore_60d"], "is_new": true}, {"model_id": "new_h7_STRESS_LightGBM_N5_t6", "algo": "LightGBM", "regime": "STRESS", "horizon": 7, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "HUM_Humana_ret_5d", "BTI_BritishAmerican_ret_20d", "CLX_Clorox_vol_20d"], "is_new": true}, {"model_id": "new_h7_STRESS_LightGBM_N5_t7", "algo": "LightGBM", "regime": "STRESS", "horizon": 7, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWM_Malaysia_zscore_60d", "AMD_ret_1d", "EWJ_Japan_vol_20d"], "is_new": true}, {"model_id": "new_h7_STRESS_LightGBM_N8_t0", "algo": "LightGBM", "regime": "STRESS", "horizon": 7, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "US5Y_Rate_ret_5d", "MSTR_Bitcoin3_ret_5d", "TGT_Target_zscore_60d", "CI_Cigna_vol_20d", "IBEX_Spain_ret_20d", "Nikkei_Japan_vol_20d"], "is_new": true}, {"model_id": "new_h7_STRESS_LightGBM_N8_t1", "algo": "LightGBM", "regime": "STRESS", "horizon": 7, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "MSTR_Bitcoin3_ret_5d", "FedFunds_zscore_60d", "EWQ_France_ret_20d", "DIS_vol_20d", "M_Macys_vol_20d", "EWG_Germany_vol_20d"], "is_new": true}, {"model_id": "new_h7_STRESS_LightGBM_N8_t2", "algo": "LightGBM", "regime": "STRESS", "horizon": 7, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWL_Switzerland_zscore_60d", "TM_Telephone_ret_1d", "DHR_vol_20d", "heston_ev_h3", "DOW_Price_zscore_60d", "EXC_Exelon_ret_1d"], "is_new": true}, {"model_id": "new_h7_STRESS_LightGBM_N8_t3", "algo": "LightGBM", "regime": "STRESS", "horizon": 7, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "CPB_CampbellSoup_vol_20d", "MRK_Merck_zscore_60d", "EWQ_France_zscore_60d", "TXN_vol_20d", "HangSeng_HK_ret_1d", "CPB_CampbellSoup_ret_20d"], "is_new": true}, {"model_id": "new_h7_STRESS_LightGBM_N8_t4", "algo": "LightGBM", "regime": "STRESS", "horizon": 7, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "WTI_Oil_FRED_zscore_60d", "SBUX_ret_5d", "INTC_ret_1d", "ES_Evergy_ret_1d", "MS_MorganStanley_zscore_60d", "AMZN_ret_5d"], "is_new": true}, {"model_id": "new_h7_STRESS_LightGBM_N8_t5", "algo": "LightGBM", "regime": "STRESS", "horizon": 7, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "GD_GeneralDynamics_zscore_60d", "EWL_Switzerland_vol_20d", "DAX_Germany_vol_20d", "CPB_CampbellSoup_ret_5d", "EWM_Malaysia_ret_1d", "heston_var_ev_h3"], "is_new": true}, {"model_id": "new_h7_STRESS_LightGBM_N8_t6", "algo": "LightGBM", "regime": "STRESS", "horizon": 7, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "ORCL_zscore_60d", "Retail_Sales_zscore_60d", "HUM_Humana_ret_5d", "BLK_BlackRock_zscore_60d", "vix_acceleration_1d", "XOM_ret_20d"], "is_new": true}, {"model_id": "new_h7_STRESS_LightGBM_N8_t7", "algo": "LightGBM", "regime": "STRESS", "horizon": 7, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "SLB_Schlumberger_ret_5d", "LOW_Lowes_ret_20d", "DE_Deere_ret_5d", "EOG_EOGResources_vol_20d", "ASX_Australia_vol_20d", "IBEX_Spain_ret_20d"], "is_new": true}, {"model_id": "new_h7_STRESS_LightGBM_N10_t0", "algo": "LightGBM", "regime": "STRESS", "horizon": 7, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "MS_MorganStanley_ret_1d", "EOG_EOGResources_vol_20d", "EWM_Malaysia_zscore_60d", "BLK_BlackRock_zscore_60d", "INTC_ret_5d", "AORD_AUS_zscore_60d", "GD_GeneralDynamics_zscore_60d", "ORCL_zscore_60d"], "is_new": true}, {"model_id": "new_h7_STRESS_LightGBM_N10_t1", "algo": "LightGBM", "regime": "STRESS", "horizon": 7, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWC_Canada_zscore_60d", "HangSeng_HK_vol_20d", "LLY_zscore_60d", "PG_ret_20d", "EWG_Germany_ret_20d", "heston_ev_h3", "HD_ret_20d", "NWL_Newell_ret_20d"], "is_new": true}, {"model_id": "new_h7_STRESS_LightGBM_N10_t2", "algo": "LightGBM", "regime": "STRESS", "horizon": 7, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "SBUX_ret_5d", "spx_abs_ret_max_5d", "vix_mean_abs_ret_5d", "IBEX_Spain_ret_20d", "XLY_Disc_vol_20d", "WTI_Oil_FRED_zscore_60d", "US1Y_Rate_ret_20d", "SPY_zscore_60d"], "is_new": true}, {"model_id": "new_h7_STRESS_LightGBM_N10_t3", "algo": "LightGBM", "regime": "STRESS", "horizon": 7, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "PAYX_Paychex_zscore_60d", "PAYX_Paychex_ret_20d", "EWY_Korea_ret_20d", "PLD_Prologis_ret_5d", "Nikkei_Japan_zscore_60d", "Michigan_Sentiment_ret_20d", "AMD_ret_5d", "Nikkei_Japan_vol_20d"], "is_new": true}, {"model_id": "new_h7_STRESS_LightGBM_N10_t4", "algo": "LightGBM", "regime": "STRESS", "horizon": 7, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "SBUX_zscore_60d", "EFFR_vol_20d", "HD_ret_5d", "INTC_ret_5d", "PAYX_Paychex_zscore_60d", "HD_ret_20d", "FedFunds_zscore_60d", "Core_PCE_zscore_60d"], "is_new": true}, {"model_id": "new_h7_STRESS_LightGBM_N10_t5", "algo": "LightGBM", "regime": "STRESS", "horizon": 7, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "ASX_Australia_ret_5d", "TED_Spread_vol_20d", "NEE_NextEra_ret_20d", "AXP_Amex_ret_20d", "HangSeng_HK_vol_20d", "SLB_Schlumberger_ret_5d", "VVIX_ret_20d", "US5Y_Rate_ret_5d"], "is_new": true}, {"model_id": "new_h7_STRESS_LightGBM_N10_t6", "algo": "LightGBM", "regime": "STRESS", "horizon": 7, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "CLX_Clorox_vol_20d", "HUM_Humana_ret_5d", "US6M_Rate_ret_20d", "NWL_Newell_ret_20d", "TXN_vol_20d", "PFE_ret_1d", "EWA_Australia_zscore_60d", "MS_MorganStanley_ret_5d"], "is_new": true}, {"model_id": "new_h7_STRESS_LightGBM_N10_t7", "algo": "LightGBM", "regime": "STRESS", "horizon": 7, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "CLX_Clorox_vol_20d", "SPY_zscore_60d", "XLF_Fin_vol_20d", "vix_mean_abs_ret_5d", "BTI_BritishAmerican_ret_20d", "EWQ_France_ret_20d", "HD_ret_5d", "XLY_Disc_vol_20d"], "is_new": true}, {"model_id": "new_h7_STRESS_LightGBM_N12_t0", "algo": "LightGBM", "regime": "STRESS", "horizon": 7, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "PPL_PPL_ret_1d", "MO_AltriaMG_ret_1d", "ITT_ITTInc_ret_5d", "AORD_AUS_zscore_60d", "heston_var_ev_h5", "EWH_HongKong_ret_5d", "spx_momentum_3d", "CPB_CampbellSoup_zscore_60d", "3M_ret_5d", "XLK_Tech_zscore_60d"], "is_new": true}, {"model_id": "new_h7_STRESS_LightGBM_N12_t1", "algo": "LightGBM", "regime": "STRESS", "horizon": 7, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWA_Australia_zscore_60d", "PCAR_PaccarInc_ret_5d", "XLY_Disc_vol_20d", "QQQ_vol_20d", "EWM_Malaysia_vol_20d", "M_Macys_vol_20d", "SBUX_vol_20d", "TED_Spread_zscore_60d", "US3M_Rate_zscore_60d", "vix_acceleration_1d"], "is_new": true}, {"model_id": "new_h7_STRESS_LightGBM_N12_t2", "algo": "LightGBM", "regime": "STRESS", "horizon": 7, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EMR_Emerson_ret_20d", "EOG_EOGResources_vol_20d", "EWG_Germany_vol_20d", "CPB_CampbellSoup_ret_20d", "FedFunds_zscore_60d", "DE_Deere_ret_5d", "CPB_CampbellSoup_vol_20d", "CPB_CampbellSoup_ret_5d", "spx_vol_5d", "EQIX_Equinix_ret_5d"], "is_new": true}, {"model_id": "new_h7_STRESS_LightGBM_N12_t3", "algo": "LightGBM", "regime": "STRESS", "horizon": 7, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "GILD_Gilead_ret_20d", "EXC_Exelon_zscore_60d", "GE_ret_1d", "gjr_condvar_h1", "HUM_Humana_ret_5d", "3M_ret_5d", "PFE_ret_1d", "DE_Deere_ret_5d", "CLX_Clorox_vol_20d", "Brent_Oil_FRED_ret_20d"], "is_new": true}, {"model_id": "new_h7_STRESS_LightGBM_N12_t4", "algo": "LightGBM", "regime": "STRESS", "horizon": 7, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "VOD_Vodafone_zscore_60d", "MSTR_Bitcoin3_ret_1d", "MS_MorganStanley_zscore_60d", "PAYX_Paychex_zscore_60d", "US3Y_Rate_ret_5d", "LMT_LockheedMartin_vol_20d", "NFCI_ret_5d", "CPB_CampbellSoup_ret_5d", "DAX_Germany_vol_20d", "EWJ_Japan_vol_20d"], "is_new": true}, {"model_id": "new_h7_STRESS_LightGBM_N12_t5", "algo": "LightGBM", "regime": "STRESS", "horizon": 7, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "AXP_Amex_ret_20d", "Brent_Oil_FRED_ret_20d", "US7Y_Rate_ret_20d", "EWL_Switzerland_vol_20d", "Michigan_Sentiment_ret_20d", "EWJ_Japan_vol_20d", "US6M_Rate_ret_20d", "IYR_US_REIT2_zscore_60d", "VRP_ma5", "SBUX_vol_20d"], "is_new": true}, {"model_id": "new_h7_STRESS_LightGBM_N12_t6", "algo": "LightGBM", "regime": "STRESS", "horizon": 7, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "SCHW_Schwab_ret_5d", "MSTR_Bitcoin3_ret_5d", "CPB_CampbellSoup_vol_20d", "EOG_EOGResources_ret_5d", "ORCL_zscore_60d", "NVDA_vol_20d", "LOW_Lowes_ret_5d", "CTAS_Cintas_vol_20d", "EXC_Exelon_ret_1d", "VVIX_ret_20d"], "is_new": true}, {"model_id": "new_h7_STRESS_LightGBM_N12_t7", "algo": "LightGBM", "regime": "STRESS", "horizon": 7, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "BLK_BlackRock_zscore_60d", "US5Y_Rate_ret_5d", "NEE_NextEra_ret_20d", "XLB_Materials_zscore_60d", "ASX_Australia_vol_20d", "AMGN_Amgen_ret_1d", "GE_ret_1d", "CI_Cigna_vol_20d", "SO_SouthernCo_ret_5d", "BTI_BritishAmerican_ret_5d"], "is_new": true}, {"model_id": "new_h7_STRESS_LightGBM_N15_t0", "algo": "LightGBM", "regime": "STRESS", "horizon": 7, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "DHR_vol_20d", "ASX_Australia_vol_20d", "NEE_NextEra_ret_20d", "MSTR_Bitcoin3_ret_5d", "Nikkei_Japan_vol_20d", "CLX_Clorox_vol_20d", "3M_vol_20d", "EOG_EOGResources_ret_5d", "AVB_AvalonBay_zscore_60d", "CCI_CrownCastle_vol_20d", "CMCSA_ret_1d", "PAYX_Paychex_zscore_60d", "GE_ret_1d"], "is_new": true}, {"model_id": "new_h7_STRESS_LightGBM_N15_t1", "algo": "LightGBM", "regime": "STRESS", "horizon": 7, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "PAYX_Paychex_vol_20d", "PFE_ret_1d", "EWL_Switzerland_zscore_60d", "Brent_Oil_FRED_ret_20d", "BDX_Becton_Dickinson_ret_20d", "CPB_CampbellSoup_zscore_60d", "CCI_CrownCastle_vol_20d", "AORD_AUS_zscore_60d", "Michigan_Sentiment_ret_20d", "BA_ret_1d", "3M_ret_5d", "EWM_Malaysia_zscore_60d", "3M_vol_20d"], "is_new": true}, {"model_id": "new_h7_STRESS_LightGBM_N15_t2", "algo": "LightGBM", "regime": "STRESS", "horizon": 7, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "SBUX_ret_5d", "AXP_Amex_ret_20d", "MS_MorganStanley_ret_1d", "Core_CPI_zscore_60d", "EOG_EOGResources_vol_20d", "HD_ret_1d", "IWM_SmallCap_vol_20d", "MO_AltriaMG_ret_1d", "LMT_LockheedMartin_vol_20d", "US6M_Rate_ret_20d", "LUV_SouthwestAir_ret_5d", "BTI_BritishAmerican_ret_20d", "EWL_Switzerland_zscore_60d"], "is_new": true}, {"model_id": "new_h7_STRESS_LightGBM_N15_t3", "algo": "LightGBM", "regime": "STRESS", "horizon": 7, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "NVDA_vol_20d", "Nikkei_Japan_zscore_60d", "IYR_US_REIT2_zscore_60d", "AVB_AvalonBay_zscore_60d", "XLB_Materials_zscore_60d", "TM_Telephone_vol_20d", "AMZN_ret_5d", "DAX_Germany_vol_20d", "Retail_Sales_zscore_60d", "heston_ev_h3", "BA_ret_1d", "XOM_ret_20d", "LMT_LockheedMartin_vol_20d"], "is_new": true}, {"model_id": "new_h7_STRESS_LightGBM_N15_t4", "algo": "LightGBM", "regime": "STRESS", "horizon": 7, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "IWM_SmallCap_vol_20d", "PAYX_Paychex_ret_20d", "TXN_vol_20d", "US1Y_Rate_ret_20d", "GILD_Gilead_ret_20d", "SBUX_zscore_60d", "Core_PCE_zscore_60d", "EXC_Exelon_zscore_60d", "gjr_condvar_h1", "EWH_HongKong_ret_5d", "LMT_LockheedMartin_vol_20d", "Brent_Oil_FRED_ret_20d", "EOG_EOGResources_vol_20d"], "is_new": true}, {"model_id": "new_h7_STRESS_LightGBM_N15_t5", "algo": "LightGBM", "regime": "STRESS", "horizon": 7, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "DAX_Germany_vol_20d", "INTC_ret_1d", "SLB_Schlumberger_ret_5d", "EQIX_Equinix_ret_5d", "T_ret_1d", "IBEX_Spain_ret_20d", "EWL_Switzerland_zscore_60d", "MS_MorganStanley_ret_5d", "IYM_BasicMaterials_ret_20d", "ORCL_vol_20d", "SLB_Schlumberger_ret_1d", "MO_AltriaMG_ret_1d", "MS_MorganStanley_ret_1d"], "is_new": true}, {"model_id": "new_h7_STRESS_LightGBM_N15_t6", "algo": "LightGBM", "regime": "STRESS", "horizon": 7, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWM_Malaysia_vol_20d", "AORD_AUS_zscore_60d", "DOW_Price_zscore_60d", "PAYX_Paychex_zscore_60d", "SO_SouthernCo_ret_5d", "XLV_Health_zscore_60d", "CPB_CampbellSoup_ret_20d", "Brent_Oil_FRED_ret_5d", "PFE_ret_1d", "HangSeng_HK_vol_20d", "IYR_US_REIT2_zscore_60d", "ORCL_vol_20d", "CPB_CampbellSoup_zscore_60d"], "is_new": true}, {"model_id": "new_h7_STRESS_LightGBM_N15_t7", "algo": "LightGBM", "regime": "STRESS", "horizon": 7, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "AMT_AmericanTower_ret_1d", "MRK_Merck_zscore_60d", "FedFunds_zscore_60d", "ASX_Australia_vol_20d", "GD_GeneralDynamics_zscore_60d", "SLB_Schlumberger_ret_5d", "EWC_Canada_zscore_60d", "IBEX_Spain_ret_20d", "LMT_LockheedMartin_ret_1d", "heston_var_ev_h3", "PG_ret_20d", "NWL_Newell_ret_20d", "TM_Telephone_ret_1d"], "is_new": true}, {"model_id": "new_h7_STRESS_LightGBM_N20_t0", "algo": "LightGBM", "regime": "STRESS", "horizon": 7, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWM_Malaysia_ret_1d", "EOG_EOGResources_ret_5d", "Brent_Oil_FRED_ret_20d", "HD_ret_20d", "Nikkei_Japan_vol_20d", "ASX_Australia_vol_20d", "Retail_Sales_zscore_60d", "QQQ_vol_20d", "US30Y_Rate_ret_20d", "Brent_Oil_FRED_ret_5d", "EWG_Germany_vol_20d", "XLB_Materials_zscore_60d", "XLV_Health_zscore_60d", "MO_AltriaMG_ret_1d", "heston_var_ev_h7", "ORCL_zscore_60d", "T_ret_1d", "EXC_Exelon_zscore_60d"], "is_new": true}, {"model_id": "new_h7_STRESS_LightGBM_N20_t1", "algo": "LightGBM", "regime": "STRESS", "horizon": 7, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "JNJ_ret_1d", "HangSeng_HK_ret_1d", "SJM_JM_Smucker_ret_1d", "MSTR_Bitcoin3_ret_1d", "SCHW_Schwab_ret_5d", "FedFunds_zscore_60d", "GD_GeneralDynamics_zscore_60d", "hmm_p_stress", "Industrial_Production_zscore_60d", "EWJ_Japan_vol_20d", "CCI_CrownCastle_vol_20d", "EWS_Singapore_ret_5d", "SJM_JM_Smucker_ret_5d", "NOC_Northrop_ret_20d", "Retail_Sales_zscore_60d", "T10Y2Y_Spread_ret_5d", "DAX_Germany_zscore_60d", "EWG_Germany_ret_20d"], "is_new": true}, {"model_id": "new_h7_STRESS_LightGBM_N20_t2", "algo": "LightGBM", "regime": "STRESS", "horizon": 7, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "DIS_vol_20d", "heston_ev_h3", "EWG_Germany_ret_20d", "QQQ_vol_20d", "spx_vol_5d", "HUM_Humana_ret_5d", "AVB_AvalonBay_zscore_60d", "INTC_ret_1d", "gjr_condvar_h1", "CMCSA_ret_1d", "Brent_Oil_FRED_ret_20d", "ORCL_zscore_60d", "LOW_Lowes_ret_5d", "SBUX_ret_5d", "spx_abs_ret_max_5d", "EWM_Malaysia_zscore_60d", "EFFR_vol_20d", "US1Y_Rate_ret_20d"], "is_new": true}, {"model_id": "new_h7_STRESS_LightGBM_N20_t3", "algo": "LightGBM", "regime": "STRESS", "horizon": 7, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "T10Y2Y_Spread_ret_5d", "CTAS_Cintas_vol_20d", "HangSeng_HK_vol_20d", "EQR_Equity_ret_1d", "AMGN_Amgen_ret_1d", "BA_ret_1d", "DE_Deere_ret_5d", "TXN_vol_20d", "SBUX_ret_5d", "spx_momentum_3d", "EWS_Singapore_ret_5d", "HD_ret_20d", "3M_vol_20d", "EWG_Germany_ret_20d", "NEE_NextEra_ret_20d", "XOM_ret_20d", "IYR_US_REIT2_zscore_60d", "MS_MorganStanley_ret_1d"], "is_new": true}, {"model_id": "new_h7_STRESS_LightGBM_N20_t4", "algo": "LightGBM", "regime": "STRESS", "horizon": 7, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "HUM_Humana_ret_5d", "DAX_Germany_zscore_60d", "VVIX_ret_20d", "EWL_Switzerland_zscore_60d", "DIS_vol_20d", "Nikkei_Japan_vol_20d", "PFE_ret_1d", "EXC_Exelon_zscore_60d", "IYM_BasicMaterials_ret_20d", "EFFR_ret_1d", "BA_ret_1d", "SPY_zscore_60d", "DE_Deere_vol_20d", "NWL_Newell_ret_20d", "AMZN_ret_5d", "SBUX_ret_5d", "LMT_LockheedMartin_ret_1d", "SO_SouthernCo_ret_5d"], "is_new": true}, {"model_id": "new_h7_STRESS_LightGBM_N20_t5", "algo": "LightGBM", "regime": "STRESS", "horizon": 7, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "Brent_Oil_FRED_ret_5d", "CPB_CampbellSoup_ret_5d", "ORCL_vol_20d", "AMZN_ret_5d", "CCI_CrownCastle_vol_20d", "DAX_Germany_zscore_60d", "HD_ret_5d", "AMD_ret_5d", "EWA_Australia_ret_1d", "Nikkei_Japan_vol_20d", "BA_ret_1d", "EWC_Canada_zscore_60d", "NVDA_vol_20d", "AMD_ret_1d", "TM_Telephone_vol_20d", "DHR_vol_20d", "PPL_PPL_ret_1d", "SPY_zscore_60d"], "is_new": true}, {"model_id": "new_h7_STRESS_LightGBM_N20_t6", "algo": "LightGBM", "regime": "STRESS", "horizon": 7, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "TED_Spread_vol_20d", "GILD_Gilead_ret_20d", "US7Y_Rate_ret_20d", "US30Y_Rate_ret_20d", "QQQ_vol_20d", "LOW_Lowes_ret_20d", "LMT_LockheedMartin_vol_20d", "gjr_condvar_h1", "EFFR_vol_20d", "T10Y2Y_Spread_ret_5d", "BTI_BritishAmerican_ret_5d", "AXP_Amex_ret_20d", "spx_momentum_3d", "EWH_HongKong_ret_5d", "CPB_CampbellSoup_zscore_60d", "XLV_Health_zscore_60d", "XLB_Materials_zscore_60d", "spx_abs_ret_max_5d"], "is_new": true}, {"model_id": "new_h7_STRESS_LightGBM_N20_t7", "algo": "LightGBM", "regime": "STRESS", "horizon": 7, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "MRK_Merck_zscore_60d", "ES_Evergy_ret_1d", "EWM_Malaysia_zscore_60d", "CMCSA_ret_1d", "EWM_Malaysia_vol_20d", "EWQ_France_zscore_60d", "IYM_BasicMaterials_ret_20d", "SPY_zscore_60d", "NVDA_vol_20d", "EQIX_Equinix_ret_5d", "M_Macys_vol_20d", "SLB_Schlumberger_ret_1d", "LLY_zscore_60d", "AMGN_Amgen_ret_1d", "TM_Telephone_ret_1d", "IWM_SmallCap_vol_20d", "gjr_condvar_h1", "AMD_ret_1d"], "is_new": true}, {"model_id": "new_h7_STRESS_LightGBM_N25_t0", "algo": "LightGBM", "regime": "STRESS", "horizon": 7, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "CPB_CampbellSoup_zscore_60d", "AXP_Amex_vol_20d", "CCI_CrownCastle_vol_20d", "Michigan_Sentiment_ret_20d", "EWG_Germany_vol_20d", "DIS_vol_20d", "XOM_ret_1d", "BDX_Becton_Dickinson_ret_20d", "3M_ret_5d", "vix_acceleration_1d", "IBEX_Spain_ret_20d", "ENB_EnbridgeInc_ret_1d", "CPB_CampbellSoup_vol_20d", "XLV_Health_zscore_60d", "VOD_Vodafone_zscore_60d", "TM_Telephone_ret_1d", "US3M_Rate_zscore_60d", "EWL_Switzerland_vol_20d", "VVIX_ret_20d", "MS_MorganStanley_ret_1d", "AVB_AvalonBay_zscore_60d", "SBUX_zscore_60d", "EWA_Australia_zscore_60d"], "is_new": true}, {"model_id": "new_h7_STRESS_LightGBM_N25_t1", "algo": "LightGBM", "regime": "STRESS", "horizon": 7, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "NVDA_vol_20d", "MS_MorganStanley_ret_5d", "Brent_Oil_FRED_ret_20d", "FedFunds_zscore_60d", "DOW_Price_zscore_60d", "AXP_Amex_ret_20d", "MSTR_Bitcoin3_ret_5d", "DAX_Germany_zscore_60d", "CTAS_Cintas_vol_20d", "spx_momentum_3d", "ASX_Australia_vol_20d", "EFFR_ret_1d", "GD_GeneralDynamics_zscore_60d", "US5Y_Rate_ret_5d", "VRP_ma5", "SPY_zscore_60d", "TXN_vol_20d", "US3M_Rate_zscore_60d", "EWY_Korea_ret_20d", "HD_zscore_60d", "heston_var_ev_h5", "CPB_CampbellSoup_zscore_60d", "SCHW_Schwab_ret_5d"], "is_new": true}, {"model_id": "new_h7_STRESS_LightGBM_N25_t2", "algo": "LightGBM", "regime": "STRESS", "horizon": 7, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "DOW_Price_zscore_60d", "EWC_Canada_zscore_60d", "DAX_Germany_vol_20d", "EQR_Equity_ret_1d", "TM_Telephone_ret_1d", "hmm_p_stress", "HUM_Humana_ret_5d", "CMCSA_ret_1d", "LUV_SouthwestAir_ret_5d", "heston_ev_h3", "spx_momentum_3d", "PFE_ret_1d", "Retail_Sales_zscore_60d", "MSTR_Bitcoin3_ret_1d", "LOW_Lowes_ret_20d", "XLV_Health_zscore_60d", "US3M_Rate_zscore_60d", "gjr_condvar_h1", "LOW_Lowes_ret_5d", "EWY_Korea_ret_20d", "EFFR_ret_1d", "GD_GeneralDynamics_zscore_60d", "ENB_EnbridgeInc_ret_1d"], "is_new": true}, {"model_id": "new_h7_STRESS_LightGBM_N25_t3", "algo": "LightGBM", "regime": "STRESS", "horizon": 7, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "US3Y_Rate_ret_5d", "ITT_ITTInc_ret_5d", "PPL_PPL_ret_1d", "MS_MorganStanley_ret_1d", "EOG_EOGResources_vol_20d", "HangSeng_HK_ret_1d", "US6M_Rate_ret_20d", "EQIX_Equinix_ret_5d", "VVIX_ret_20d", "TXN_vol_20d", "XOM_ret_1d", "NVDA_vol_20d", "CPB_CampbellSoup_zscore_60d", "T10Y2Y_Spread_ret_5d", "PFE_ret_1d", "IWM_SmallCap_vol_20d", "ENB_EnbridgeInc_ret_1d", "SPY_zscore_60d", "US30Y_Rate_ret_20d", "EQR_Equity_ret_1d", "XLB_Materials_zscore_60d", "DAX_Germany_vol_20d", "PAYX_Paychex_ret_20d"], "is_new": true}, {"model_id": "new_h7_STRESS_LightGBM_N25_t4", "algo": "LightGBM", "regime": "STRESS", "horizon": 7, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "GD_GeneralDynamics_zscore_60d", "INTC_ret_1d", "TXN_vol_20d", "LOW_Lowes_ret_20d", "DAX_Germany_zscore_60d", "DOW_Price_zscore_60d", "HUM_Humana_ret_5d", "EWM_Malaysia_ret_1d", "CCI_CrownCastle_vol_20d", "Nikkei_Japan_zscore_60d", "PAYX_Paychex_vol_20d", "DE_Deere_ret_5d", "BA_ret_1d", "T_ret_1d", "EWM_Malaysia_vol_20d", "US1Y_Rate_ret_20d", "SPY_zscore_60d", "XLF_Fin_vol_20d", "HangSeng_HK_ret_1d", "Nikkei_Japan_vol_20d", "CLX_Clorox_vol_20d", "AMT_AmericanTower_ret_1d", "T10Y2Y_Spread_ret_5d"], "is_new": true}, {"model_id": "new_h7_STRESS_LightGBM_N25_t5", "algo": "LightGBM", "regime": "STRESS", "horizon": 7, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "spx_abs_ret_max_5d", "ITT_ITTInc_ret_5d", "XLF_Fin_vol_20d", "CCI_CrownCastle_vol_20d", "WTI_Oil_FRED_zscore_60d", "HD_ret_20d", "AMD_ret_5d", "NOC_Northrop_ret_20d", "ORCL_vol_20d", "XOM_ret_1d", "EXC_Exelon_ret_1d", "DE_Deere_vol_20d", "IWM_SmallCap_vol_20d", "ASX_Australia_ret_5d", "MO_AltriaMG_ret_1d", "IYR_US_REIT2_zscore_60d", "MSTR_Bitcoin3_ret_20d", "DOW_Price_zscore_60d", "US6M_Rate_ret_20d", "PAYX_Paychex_vol_20d", "Industrial_Production_zscore_60d", "MRK_Merck_zscore_60d", "EWM_Malaysia_vol_20d"], "is_new": true}, {"model_id": "new_h7_STRESS_LightGBM_N25_t6", "algo": "LightGBM", "regime": "STRESS", "horizon": 7, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "ENB_EnbridgeInc_ret_1d", "Core_CPI_zscore_60d", "XLF_Fin_vol_20d", "SLB_Schlumberger_ret_5d", "LLY_zscore_60d", "XOM_ret_20d", "Core_PCE_zscore_60d", "Industrial_Production_zscore_60d", "MRK_Merck_zscore_60d", "gjr_condvar_h1", "EWA_Australia_ret_1d", "US1Y_Rate_ret_5d", "SO_SouthernCo_ret_5d", "HD_ret_5d", "Nikkei_Japan_zscore_60d", "US3Y_Rate_ret_5d", "PAYX_Paychex_zscore_60d", "LMT_LockheedMartin_ret_1d", "EWQ_France_zscore_60d", "T10Y2Y_Spread_ret_5d", "HUM_Humana_ret_5d", "PG_ret_20d", "EWL_Switzerland_vol_20d"], "is_new": true}, {"model_id": "new_h7_STRESS_LightGBM_N25_t7", "algo": "LightGBM", "regime": "STRESS", "horizon": 7, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "Nikkei_Japan_zscore_60d", "vix_mean_abs_ret_5d", "BTI_BritishAmerican_ret_20d", "DAX_Germany_vol_20d", "EWG_Germany_ret_20d", "CPB_CampbellSoup_vol_20d", "Retail_Sales_zscore_60d", "MSTR_Bitcoin3_ret_5d", "spx_momentum_3d", "IYM_BasicMaterials_ret_20d", "XOM_ret_20d", "SJM_JM_Smucker_ret_5d", "NOC_Northrop_ret_20d", "LUV_SouthwestAir_ret_5d", "3M_ret_5d", "GE_ret_1d", "SLB_Schlumberger_ret_1d", "PPL_PPL_ret_1d", "BDX_Becton_Dickinson_ret_20d", "US1Y_Rate_ret_5d", "PAYX_Paychex_vol_20d", "EMR_Emerson_ret_20d", "T10Y2Y_Spread_ret_5d"], "is_new": true}, {"model_id": "new_h7_STRESS_LightGBM_N30_t0", "algo": "LightGBM", "regime": "STRESS", "horizon": 7, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "AMD_ret_5d", "ORCL_vol_20d", "PLD_Prologis_ret_5d", "MS_MorganStanley_ret_5d", "GILD_Gilead_ret_20d", "gjr_condvar_h1", "LMT_LockheedMartin_ret_1d", "spx_abs_ret_max_5d", "ASX_Australia_ret_5d", "LOW_Lowes_ret_20d", "NWL_Newell_ret_20d", "AXP_Amex_ret_20d", "ENB_EnbridgeInc_ret_1d", "BDX_Becton_Dickinson_ret_20d", "BTI_BritishAmerican_ret_20d", "TED_Spread_vol_20d", "PAYX_Paychex_ret_20d", "EWG_Germany_vol_20d", "EFFR_ret_1d", "TM_Telephone_ret_1d", "IYR_US_REIT2_zscore_60d", "MS_MorganStanley_zscore_60d", "IWM_SmallCap_vol_20d", "US1Y_Rate_ret_5d", "BLK_BlackRock_zscore_60d", "XLY_Disc_vol_20d", "heston_var_ev_h7", "EWQ_France_ret_20d"], "is_new": true}, {"model_id": "new_h7_STRESS_LightGBM_N30_t1", "algo": "LightGBM", "regime": "STRESS", "horizon": 7, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "GD_GeneralDynamics_zscore_60d", "NVDA_vol_20d", "IYM_BasicMaterials_ret_20d", "AORD_AUS_zscore_60d", "EWG_Germany_ret_20d", "DOW_Price_zscore_60d", "HD_zscore_60d", "AMZN_ret_5d", "EWG_Germany_vol_20d", "heston_var_ev_h5", "QQQ_vol_20d", "AMD_ret_1d", "EWM_Malaysia_ret_1d", "AXP_Amex_ret_20d", "US1Y_Rate_ret_20d", "SJM_JM_Smucker_ret_5d", "Industrial_Production_zscore_60d", "GILD_Gilead_ret_20d", "ES_Evergy_ret_1d", "EWM_Malaysia_zscore_60d", "IBEX_Spain_ret_20d", "SPY_zscore_60d", "TED_Spread_vol_20d", "EWL_Switzerland_vol_20d", "PPL_PPL_ret_1d", "NFCI_ret_5d", "EWQ_France_ret_20d", "XLK_Tech_zscore_60d"], "is_new": true}, {"model_id": "new_h7_STRESS_LightGBM_N30_t2", "algo": "LightGBM", "regime": "STRESS", "horizon": 7, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "SPY_zscore_60d", "XLK_Tech_zscore_60d", "SBUX_zscore_60d", "XOM_ret_20d", "EWA_Australia_ret_1d", "MS_MorganStanley_zscore_60d", "JNJ_ret_1d", "HD_zscore_60d", "spx_vol_5d", "ORCL_zscore_60d", "EWY_Korea_zscore_60d", "NFCI_ret_5d", "BLK_BlackRock_zscore_60d", "CCI_CrownCastle_vol_20d", "CI_Cigna_vol_20d", "ASX_Australia_ret_5d", "US30Y_Rate_ret_20d", "GD_GeneralDynamics_zscore_60d", "spx_abs_ret_max_5d", "US7Y_Rate_ret_20d", "BA_ret_1d", "XLV_Health_zscore_60d", "NWL_Newell_ret_20d", "EWY_Korea_ret_20d", "LMT_LockheedMartin_vol_20d", "vix_mean_abs_ret_5d", "SJM_JM_Smucker_ret_1d", "US5Y_Rate_ret_5d"], "is_new": true}, {"model_id": "new_h7_STRESS_LightGBM_N30_t3", "algo": "LightGBM", "regime": "STRESS", "horizon": 7, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "BA_ret_1d", "DHR_ret_1d", "INTC_ret_1d", "EWG_Germany_ret_20d", "AMGN_Amgen_ret_1d", "AMD_ret_5d", "XOM_ret_1d", "LOW_Lowes_ret_5d", "heston_ev_h3", "SO_SouthernCo_ret_5d", "HD_ret_20d", "AVB_AvalonBay_zscore_60d", "EWQ_France_zscore_60d", "GD_GeneralDynamics_zscore_60d", "HangSeng_HK_vol_20d", "US5Y_Rate_ret_5d", "BTI_BritishAmerican_ret_5d", "US3M_Rate_zscore_60d", "LUV_SouthwestAir_ret_5d", "EXC_Exelon_ret_1d", "3M_ret_5d", "Core_CPI_zscore_60d", "PAYX_Paychex_zscore_60d", "PG_ret_20d", "HUM_Humana_ret_5d", "LOW_Lowes_ret_20d", "AMT_AmericanTower_ret_1d", "EWA_Australia_zscore_60d"], "is_new": true}, {"model_id": "new_h7_STRESS_LightGBM_N30_t4", "algo": "LightGBM", "regime": "STRESS", "horizon": 7, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "INTC_ret_1d", "EWM_Malaysia_ret_1d", "BA_ret_1d", "INTC_ret_5d", "T10Y2Y_Spread_ret_5d", "CMCSA_ret_1d", "EWA_Australia_ret_1d", "IWM_SmallCap_vol_20d", "MO_AltriaMG_ret_1d", "CTAS_Cintas_vol_20d", "EQR_Equity_ret_1d", "NOC_Northrop_ret_20d", "GILD_Gilead_ret_20d", "3M_vol_20d", "AMZN_ret_5d", "US30Y_Rate_ret_20d", "VVIX_ret_20d", "EWG_Germany_ret_20d", "US1Y_Rate_ret_5d", "PAYX_Paychex_vol_20d", "LMT_LockheedMartin_ret_1d", "ITT_ITTInc_ret_5d", "T_ret_1d", "MS_MorganStanley_zscore_60d", "TM_Telephone_vol_20d", "BLK_BlackRock_zscore_60d", "MS_MorganStanley_ret_5d", "Michigan_Sentiment_ret_20d"], "is_new": true}, {"model_id": "new_h7_STRESS_LightGBM_N30_t5", "algo": "LightGBM", "regime": "STRESS", "horizon": 7, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "NWL_Newell_ret_20d", "PAYX_Paychex_zscore_60d", "EWA_Australia_zscore_60d", "SPY_zscore_60d", "SBUX_vol_20d", "INTC_ret_1d", "AVB_AvalonBay_zscore_60d", "DOW_Price_zscore_60d", "Retail_Sales_zscore_60d", "TXN_vol_20d", "ORCL_zscore_60d", "spx_vol_5d", "MS_MorganStanley_ret_1d", "EMR_Emerson_ret_20d", "CPB_CampbellSoup_ret_20d", "T_ret_1d", "HUM_Humana_ret_5d", "SLB_Schlumberger_ret_5d", "Brent_Oil_FRED_ret_20d", "PAYX_Paychex_vol_20d", "PFE_ret_1d", "EXC_Exelon_ret_1d", "IYR_US_REIT2_zscore_60d", "INTC_ret_5d", "TED_Spread_zscore_60d", "HD_zscore_60d", "US30Y_Rate_ret_20d", "Core_PCE_zscore_60d"], "is_new": true}, {"model_id": "new_h7_STRESS_LightGBM_N30_t6", "algo": "LightGBM", "regime": "STRESS", "horizon": 7, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "SBUX_ret_5d", "EWQ_France_zscore_60d", "hmm_p_stress", "GILD_Gilead_ret_20d", "SCHW_Schwab_ret_5d", "HD_zscore_60d", "EQIX_Equinix_ret_5d", "ES_Evergy_ret_1d", "Brent_Oil_FRED_ret_20d", "US6M_Rate_ret_20d", "XLB_Materials_zscore_60d", "BTI_BritishAmerican_ret_5d", "HD_ret_5d", "EWC_Canada_zscore_60d", "CPB_CampbellSoup_ret_5d", "LOW_Lowes_ret_20d", "3M_ret_5d", "US3Y_Rate_ret_5d", "ORCL_zscore_60d", "AXP_Amex_ret_20d", "vix_mean_abs_ret_5d", "CPB_CampbellSoup_zscore_60d", "VVIX_ret_20d", "EFFR_ret_1d", "AVB_AvalonBay_zscore_60d", "heston_ev_h3", "EWL_Switzerland_vol_20d", "HD_ret_20d"], "is_new": true}, {"model_id": "new_h7_STRESS_LightGBM_N30_t7", "algo": "LightGBM", "regime": "STRESS", "horizon": 7, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "XLK_Tech_zscore_60d", "Michigan_Sentiment_ret_20d", "CI_Cigna_vol_20d", "T_ret_1d", "EWM_Malaysia_vol_20d", "AMGN_Amgen_ret_1d", "EWM_Malaysia_zscore_60d", "SLB_Schlumberger_ret_5d", "NOC_Northrop_ret_20d", "EOG_EOGResources_vol_20d", "spx_momentum_3d", "US6M_Rate_ret_20d", "US30Y_Rate_ret_20d", "Core_PCE_zscore_60d", "XOM_ret_1d", "vix_mean_abs_ret_5d", "XLB_Materials_zscore_60d", "heston_var_ev_h7", "EFFR_vol_20d", "TXN_vol_20d", "EXC_Exelon_ret_1d", "AVB_AvalonBay_zscore_60d", "BDX_Becton_Dickinson_ret_20d", "PG_ret_20d", "SO_SouthernCo_ret_5d", "DHR_ret_1d", "heston_var_ev_h3", "EWH_HongKong_ret_5d"], "is_new": true}, {"model_id": "new_h7_STRESS_GradientBoosting_N5_t0", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 7, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "CPB_CampbellSoup_vol_20d", "VOD_Vodafone_zscore_60d", "CPB_CampbellSoup_ret_20d"], "is_new": true}, {"model_id": "new_h7_STRESS_GradientBoosting_N5_t1", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 7, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "LLY_zscore_60d", "VOD_Vodafone_zscore_60d", "CPB_CampbellSoup_ret_20d"], "is_new": true}, {"model_id": "new_h7_STRESS_GradientBoosting_N5_t2", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 7, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "Industrial_Production_zscore_60d", "EWJ_Japan_vol_20d", "HangSeng_HK_ret_1d"], "is_new": true}, {"model_id": "new_h7_STRESS_GradientBoosting_N5_t3", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 7, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "MO_AltriaMG_ret_1d", "TM_Telephone_vol_20d", "WTI_Oil_FRED_zscore_60d"], "is_new": true}, {"model_id": "new_h7_STRESS_GradientBoosting_N5_t4", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 7, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "AXP_Amex_ret_20d", "EWM_Malaysia_vol_20d", "BA_ret_1d"], "is_new": true}, {"model_id": "new_h7_STRESS_GradientBoosting_N5_t5", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 7, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "AMT_AmericanTower_ret_1d", "EWG_Germany_ret_20d", "EWY_Korea_zscore_60d"], "is_new": true}, {"model_id": "new_h7_STRESS_GradientBoosting_N5_t6", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 7, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "BDX_Becton_Dickinson_ret_20d", "HangSeng_HK_ret_1d", "EWH_HongKong_ret_5d"], "is_new": true}, {"model_id": "new_h7_STRESS_GradientBoosting_N5_t7", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 7, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "LOW_Lowes_ret_20d", "GE_ret_1d", "ASX_Australia_vol_20d"], "is_new": true}, {"model_id": "new_h7_STRESS_GradientBoosting_N8_t0", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 7, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "HangSeng_HK_ret_5d", "FedFunds_zscore_60d", "INTC_ret_1d", "DHR_vol_20d", "M_Macys_vol_20d", "HD_ret_5d"], "is_new": true}, {"model_id": "new_h7_STRESS_GradientBoosting_N8_t1", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 7, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "ENB_EnbridgeInc_ret_1d", "EWG_Germany_vol_20d", "CMCSA_ret_1d", "vix_mean_abs_ret_5d", "EWY_Korea_ret_20d", "GE_ret_1d"], "is_new": true}, {"model_id": "new_h7_STRESS_GradientBoosting_N8_t2", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 7, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "US30Y_Rate_ret_20d", "SO_SouthernCo_ret_5d", "HangSeng_HK_ret_1d", "GILD_Gilead_ret_20d", "TM_Telephone_ret_1d", "EWL_Switzerland_zscore_60d"], "is_new": true}, {"model_id": "new_h7_STRESS_GradientBoosting_N8_t3", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 7, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "BTI_BritishAmerican_ret_20d", "HangSeng_HK_ret_1d", "US1Y_Rate_ret_20d", "3M_vol_20d", "TED_Spread_vol_20d", "CPB_CampbellSoup_ret_5d"], "is_new": true}, {"model_id": "new_h7_STRESS_GradientBoosting_N8_t4", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 7, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWM_Malaysia_zscore_60d", "heston_var_ev_h3", "EWM_Malaysia_vol_20d", "EWH_HongKong_ret_5d", "DHR_vol_20d", "XLY_Disc_vol_20d"], "is_new": true}, {"model_id": "new_h7_STRESS_GradientBoosting_N8_t5", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 7, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "AMD_ret_1d", "DE_Deere_ret_5d", "AVB_AvalonBay_zscore_60d", "PAYX_Paychex_vol_20d", "XLK_Tech_zscore_60d", "MS_MorganStanley_ret_1d"], "is_new": true}, {"model_id": "new_h7_STRESS_GradientBoosting_N8_t6", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 7, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "TED_Spread_zscore_60d", "ASX_Australia_vol_20d", "AMT_AmericanTower_ret_1d", "TXN_vol_20d", "CPB_CampbellSoup_ret_5d", "QQQ_vol_20d"], "is_new": true}, {"model_id": "new_h7_STRESS_GradientBoosting_N8_t7", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 7, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "PLD_Prologis_ret_5d", "AVB_AvalonBay_zscore_60d", "EFFR_vol_20d", "EWQ_France_zscore_60d", "AMZN_ret_5d", "TM_Telephone_vol_20d"], "is_new": true}, {"model_id": "new_h7_STRESS_GradientBoosting_N10_t0", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 7, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "ORCL_vol_20d", "EFFR_vol_20d", "XLY_Disc_vol_20d", "SO_SouthernCo_ret_5d", "Brent_Oil_FRED_ret_20d", "VOD_Vodafone_zscore_60d", "VVIX_ret_20d", "PG_ret_20d"], "is_new": true}, {"model_id": "new_h7_STRESS_GradientBoosting_N10_t1", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 7, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWA_Australia_zscore_60d", "GILD_Gilead_ret_20d", "heston_ev_h3", "CMCSA_ret_1d", "PPL_PPL_ret_1d", "Nikkei_Japan_zscore_60d", "JNJ_ret_1d", "BLK_BlackRock_zscore_60d"], "is_new": true}, {"model_id": "new_h7_STRESS_GradientBoosting_N10_t2", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 7, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "WTI_Oil_FRED_zscore_60d", "CCI_CrownCastle_vol_20d", "XLB_Materials_zscore_60d", "EWL_Switzerland_vol_20d", "AMT_AmericanTower_ret_1d", "LMT_LockheedMartin_ret_1d", "Core_PCE_zscore_60d", "EFFR_ret_1d"], "is_new": true}, {"model_id": "new_h7_STRESS_GradientBoosting_N10_t3", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 7, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "US7Y_Rate_ret_20d", "SO_SouthernCo_ret_5d", "FedFunds_zscore_60d", "NEE_NextEra_ret_20d", "INTC_ret_5d", "DE_Deere_vol_20d", "HangSeng_HK_ret_5d", "heston_ev_h3"], "is_new": true}, {"model_id": "new_h7_STRESS_GradientBoosting_N10_t4", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 7, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "DE_Deere_vol_20d", "EWG_Germany_ret_20d", "HD_ret_1d", "EWJ_Japan_vol_20d", "BDX_Becton_Dickinson_ret_20d", "HangSeng_HK_vol_20d", "AVB_AvalonBay_zscore_60d", "MSTR_Bitcoin3_ret_5d"], "is_new": true}, {"model_id": "new_h7_STRESS_GradientBoosting_N10_t5", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 7, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "US3M_Rate_vol_20d", "MO_AltriaMG_ret_1d", "MSTR_Bitcoin3_ret_5d", "VRP_ma5", "CI_Cigna_vol_20d", "AMD_ret_5d", "EWC_Canada_zscore_60d", "EFFR_vol_20d"], "is_new": true}, {"model_id": "new_h7_STRESS_GradientBoosting_N10_t6", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 7, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "HUM_Humana_ret_5d", "PCAR_PaccarInc_ret_5d", "XLY_Disc_vol_20d", "US5Y_Rate_ret_5d", "IYR_US_REIT2_zscore_60d", "T10Y2Y_Spread_ret_5d", "Brent_Oil_FRED_ret_20d", "TM_Telephone_vol_20d"], "is_new": true}, {"model_id": "new_h7_STRESS_GradientBoosting_N10_t7", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 7, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "AMT_AmericanTower_ret_1d", "TED_Spread_vol_20d", "EWG_Germany_vol_20d", "SO_SouthernCo_ret_5d", "US5Y_Rate_ret_5d", "EOG_EOGResources_ret_5d", "CPB_CampbellSoup_ret_5d", "BTI_BritishAmerican_ret_5d"], "is_new": true}, {"model_id": "new_h7_STRESS_GradientBoosting_N12_t0", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 7, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "AMD_ret_5d", "Core_PCE_zscore_60d", "M_Macys_vol_20d", "LUV_SouthwestAir_ret_5d", "EWY_Korea_zscore_60d", "TED_Spread_vol_20d", "EWQ_France_zscore_60d", "heston_ev_h3", "SO_SouthernCo_ret_5d", "GE_ret_1d"], "is_new": true}, {"model_id": "new_h7_STRESS_GradientBoosting_N12_t1", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 7, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "CPB_CampbellSoup_ret_20d", "QQQ_vol_20d", "spx_vol_5d", "NEE_NextEra_ret_20d", "heston_var_ev_h7", "CLX_Clorox_vol_20d", "US3M_Rate_zscore_60d", "EWL_Switzerland_vol_20d", "AMD_ret_1d", "CTAS_Cintas_vol_20d"], "is_new": true}, {"model_id": "new_h7_STRESS_GradientBoosting_N12_t2", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 7, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "AXP_Amex_ret_20d", "XLY_Disc_vol_20d", "HUM_Humana_ret_5d", "vix_mean_abs_ret_5d", "VVIX_ret_20d", "SJM_JM_Smucker_ret_5d", "US3Y_Rate_ret_5d", "XLV_Health_zscore_60d", "3M_vol_20d", "vix_acceleration_1d"], "is_new": true}, {"model_id": "new_h7_STRESS_GradientBoosting_N12_t3", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 7, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "SLB_Schlumberger_ret_1d", "NFCI_ret_5d", "EXC_Exelon_zscore_60d", "XLF_Fin_vol_20d", "BLK_BlackRock_zscore_60d", "SJM_JM_Smucker_ret_1d", "XLY_Disc_vol_20d", "MS_MorganStanley_ret_1d", "Core_PCE_zscore_60d", "TM_Telephone_ret_1d"], "is_new": true}, {"model_id": "new_h7_STRESS_GradientBoosting_N12_t4", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 7, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "heston_var_ev_h7", "CPB_CampbellSoup_ret_5d", "CMCSA_ret_1d", "NOC_Northrop_ret_20d", "EWH_HongKong_ret_5d", "EWL_Switzerland_vol_20d", "GILD_Gilead_ret_20d", "spx_vol_5d", "BLK_BlackRock_zscore_60d", "DAX_Germany_vol_20d"], "is_new": true}, {"model_id": "new_h7_STRESS_GradientBoosting_N12_t5", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 7, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "CPB_CampbellSoup_ret_20d", "EWM_Malaysia_vol_20d", "WTI_Oil_FRED_zscore_60d", "EWH_HongKong_ret_5d", "EWL_Switzerland_zscore_60d", "QQQ_vol_20d", "EWA_Australia_zscore_60d", "M_Macys_vol_20d", "Brent_Oil_FRED_ret_5d", "SCHW_Schwab_ret_5d"], "is_new": true}, {"model_id": "new_h7_STRESS_GradientBoosting_N12_t6", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 7, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "MRK_Merck_zscore_60d", "TM_Telephone_vol_20d", "SBUX_zscore_60d", "HangSeng_HK_vol_20d", "hmm_p_stress", "GD_GeneralDynamics_zscore_60d", "TED_Spread_vol_20d", "JNJ_ret_1d", "EWG_Germany_ret_20d", "Core_PCE_zscore_60d"], "is_new": true}, {"model_id": "new_h7_STRESS_GradientBoosting_N12_t7", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 7, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "MRK_Merck_zscore_60d", "XLY_Disc_vol_20d", "CPB_CampbellSoup_zscore_60d", "NOC_Northrop_ret_20d", "HD_zscore_60d", "US30Y_Rate_ret_20d", "heston_var_ev_h3", "XLK_Tech_zscore_60d", "PG_ret_20d", "AMZN_ret_5d"], "is_new": true}, {"model_id": "new_h7_STRESS_GradientBoosting_N15_t0", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 7, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "BA_ret_1d", "Core_CPI_zscore_60d", "GILD_Gilead_ret_20d", "VOD_Vodafone_zscore_60d", "CCI_CrownCastle_vol_20d", "EWG_Germany_ret_20d", "EWY_Korea_ret_20d", "SBUX_vol_20d", "AXP_Amex_vol_20d", "US3M_Rate_zscore_60d", "heston_ev_h3", "GD_GeneralDynamics_zscore_60d", "PLD_Prologis_ret_5d"], "is_new": true}, {"model_id": "new_h7_STRESS_GradientBoosting_N15_t1", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 7, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "MS_MorganStanley_ret_1d", "EWQ_France_ret_20d", "BTI_BritishAmerican_ret_5d", "Industrial_Production_zscore_60d", "PLD_Prologis_ret_5d", "PG_ret_20d", "NVDA_vol_20d", "CLX_Clorox_vol_20d", "LMT_LockheedMartin_vol_20d", "HD_ret_5d", "EWL_Switzerland_zscore_60d", "LMT_LockheedMartin_ret_1d", "IWM_SmallCap_vol_20d"], "is_new": true}, {"model_id": "new_h7_STRESS_GradientBoosting_N15_t2", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 7, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EXC_Exelon_ret_1d", "WTI_Oil_FRED_zscore_60d", "EWQ_France_zscore_60d", "XOM_ret_20d", "AMGN_Amgen_ret_1d", "CPB_CampbellSoup_ret_5d", "MSTR_Bitcoin3_ret_5d", "XOM_ret_1d", "HD_ret_20d", "MRK_Merck_zscore_60d", "DHR_ret_1d", "US3M_Rate_vol_20d", "Nikkei_Japan_zscore_60d"], "is_new": true}, {"model_id": "new_h7_STRESS_GradientBoosting_N15_t3", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 7, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "HD_zscore_60d", "spx_abs_ret_max_5d", "SBUX_zscore_60d", "ASX_Australia_ret_5d", "CCI_CrownCastle_vol_20d", "IBEX_Spain_ret_20d", "XLK_Tech_zscore_60d", "EOG_EOGResources_ret_5d", "PAYX_Paychex_zscore_60d", "US3Y_Rate_ret_5d", "CPB_CampbellSoup_ret_5d", "AXP_Amex_vol_20d", "EMR_Emerson_ret_20d"], "is_new": true}, {"model_id": "new_h7_STRESS_GradientBoosting_N15_t4", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 7, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "vix_mean_abs_ret_5d", "DIS_vol_20d", "SLB_Schlumberger_ret_1d", "XLY_Disc_vol_20d", "AMT_AmericanTower_ret_1d", "MS_MorganStanley_ret_1d", "MS_MorganStanley_zscore_60d", "US3M_Rate_zscore_60d", "ASX_Australia_ret_5d", "GE_ret_1d", "NFCI_ret_5d", "EWQ_France_zscore_60d", "gjr_condvar_h1"], "is_new": true}, {"model_id": "new_h7_STRESS_GradientBoosting_N15_t5", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 7, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EXC_Exelon_ret_1d", "GE_ret_1d", "XLV_Health_zscore_60d", "PLD_Prologis_ret_5d", "XLF_Fin_vol_20d", "Nikkei_Japan_zscore_60d", "hmm_p_stress", "XOM_ret_1d", "SJM_JM_Smucker_ret_1d", "PCAR_PaccarInc_ret_5d", "heston_var_ev_h7", "CPB_CampbellSoup_ret_20d", "INTC_ret_1d"], "is_new": true}, {"model_id": "new_h7_STRESS_GradientBoosting_N15_t6", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 7, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "SBUX_vol_20d", "spx_momentum_3d", "3M_vol_20d", "AMGN_Amgen_ret_1d", "CMCSA_ret_1d", "EOG_EOGResources_vol_20d", "T_ret_1d", "AMT_AmericanTower_ret_1d", "DHR_vol_20d", "HangSeng_HK_ret_1d", "EWC_Canada_zscore_60d", "BTI_BritishAmerican_ret_5d", "US1Y_Rate_ret_5d"], "is_new": true}, {"model_id": "new_h7_STRESS_GradientBoosting_N15_t7", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 7, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "heston_var_ev_h5", "LUV_SouthwestAir_ret_5d", "EWL_Switzerland_vol_20d", "EWJ_Japan_vol_20d", "IBEX_Spain_ret_20d", "EWL_Switzerland_zscore_60d", "M_Macys_vol_20d", "Industrial_Production_zscore_60d", "XLK_Tech_zscore_60d", "HD_ret_5d", "PAYX_Paychex_vol_20d", "AMD_ret_5d", "EQIX_Equinix_ret_5d"], "is_new": true}, {"model_id": "new_h7_STRESS_GradientBoosting_N20_t0", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 7, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "vix_acceleration_1d", "EWM_Malaysia_zscore_60d", "MSTR_Bitcoin3_ret_5d", "IYM_BasicMaterials_ret_20d", "US3M_Rate_zscore_60d", "IYR_US_REIT2_zscore_60d", "NWL_Newell_ret_20d", "PG_ret_20d", "XOM_ret_1d", "US3Y_Rate_ret_5d", "Retail_Sales_zscore_60d", "BTI_BritishAmerican_ret_20d", "HD_ret_1d", "vix_mean_abs_ret_5d", "ORCL_vol_20d", "SO_SouthernCo_ret_5d", "MRK_Merck_zscore_60d", "PAYX_Paychex_zscore_60d"], "is_new": true}, {"model_id": "new_h7_STRESS_GradientBoosting_N20_t1", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 7, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "SPY_zscore_60d", "TXN_vol_20d", "heston_var_ev_h3", "EFFR_vol_20d", "INTC_ret_5d", "AXP_Amex_vol_20d", "EWL_Switzerland_zscore_60d", "EWG_Germany_vol_20d", "XOM_ret_20d", "MS_MorganStanley_zscore_60d", "PLD_Prologis_ret_5d", "AVB_AvalonBay_zscore_60d", "spx_vol_5d", "US3M_Rate_zscore_60d", "EFFR_ret_1d", "ENB_EnbridgeInc_ret_1d", "EWA_Australia_zscore_60d", "LOW_Lowes_ret_5d"], "is_new": true}, {"model_id": "new_h7_STRESS_GradientBoosting_N20_t2", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 7, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "DIS_vol_20d", "XLY_Disc_vol_20d", "SO_SouthernCo_ret_5d", "XLV_Health_zscore_60d", "SJM_JM_Smucker_ret_5d", "PAYX_Paychex_ret_20d", "DOW_Price_zscore_60d", "Industrial_Production_zscore_60d", "NFCI_ret_5d", "US7Y_Rate_ret_20d", "EWA_Australia_ret_1d", "vix_mean_abs_ret_5d", "SBUX_ret_5d", "HangSeng_HK_ret_5d", "TM_Telephone_vol_20d", "spx_vol_5d", "MSTR_Bitcoin3_ret_1d", "EWS_Singapore_ret_5d"], "is_new": true}, {"model_id": "new_h7_STRESS_GradientBoosting_N20_t3", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 7, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "INTC_ret_1d", "EWH_HongKong_ret_5d", "SBUX_vol_20d", "EQR_Equity_ret_1d", "SJM_JM_Smucker_ret_5d", "MRK_Merck_zscore_60d", "TED_Spread_vol_20d", "EWL_Switzerland_vol_20d", "US3M_Rate_zscore_60d", "EWQ_France_zscore_60d", "vix_acceleration_1d", "Brent_Oil_FRED_ret_20d", "EWJ_Japan_vol_20d", "HangSeng_HK_vol_20d", "3M_ret_5d", "EWS_Singapore_ret_5d", "MS_MorganStanley_zscore_60d", "AXP_Amex_ret_20d"], "is_new": true}, {"model_id": "new_h7_STRESS_GradientBoosting_N20_t4", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 7, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "CCI_CrownCastle_vol_20d", "JNJ_ret_1d", "AXP_Amex_vol_20d", "MS_MorganStanley_ret_1d", "Brent_Oil_FRED_ret_20d", "EWS_Singapore_ret_5d", "heston_var_ev_h7", "TED_Spread_zscore_60d", "MSTR_Bitcoin3_ret_5d", "GE_ret_1d", "VVIX_ret_20d", "AXP_Amex_ret_20d", "BA_ret_1d", "EQR_Equity_ret_1d", "SO_SouthernCo_ret_5d", "TED_Spread_vol_20d", "XOM_ret_20d", "3M_ret_5d"], "is_new": true}, {"model_id": "new_h7_STRESS_GradientBoosting_N20_t5", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 7, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "BA_ret_1d", "HD_ret_20d", "AXP_Amex_vol_20d", "HD_zscore_60d", "SCHW_Schwab_ret_5d", "XLF_Fin_vol_20d", "HangSeng_HK_ret_5d", "CLX_Clorox_vol_20d", "NWL_Newell_ret_20d", "CTAS_Cintas_vol_20d", "EFFR_ret_1d", "EWM_Malaysia_vol_20d", "NEE_NextEra_ret_20d", "MRK_Merck_zscore_60d", "EMR_Emerson_ret_20d", "TM_Telephone_vol_20d", "XOM_ret_1d", "EWC_Canada_zscore_60d"], "is_new": true}, {"model_id": "new_h7_STRESS_GradientBoosting_N20_t6", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 7, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "SCHW_Schwab_ret_5d", "INTC_ret_5d", "PAYX_Paychex_vol_20d", "EXC_Exelon_ret_1d", "HUM_Humana_ret_5d", "MO_AltriaMG_ret_1d", "TM_Telephone_vol_20d", "EMR_Emerson_ret_20d", "LMT_LockheedMartin_vol_20d", "hmm_p_stress", "heston_var_ev_h7", "PCAR_PaccarInc_ret_5d", "GD_GeneralDynamics_zscore_60d", "EOG_EOGResources_ret_5d", "EWM_Malaysia_ret_1d", "PAYX_Paychex_zscore_60d", "EQR_Equity_ret_1d", "AORD_AUS_zscore_60d"], "is_new": true}, {"model_id": "new_h7_STRESS_GradientBoosting_N20_t7", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 7, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "IYM_BasicMaterials_ret_20d", "EWH_HongKong_ret_5d", "AMGN_Amgen_ret_1d", "EWM_Malaysia_ret_1d", "XLV_Health_zscore_60d", "heston_ev_h3", "MO_AltriaMG_ret_1d", "CPB_CampbellSoup_vol_20d", "US5Y_Rate_ret_5d", "HD_ret_1d", "MS_MorganStanley_ret_1d", "TM_Telephone_ret_1d", "QQQ_vol_20d", "TED_Spread_vol_20d", "PG_ret_20d", "CTAS_Cintas_vol_20d", "EWQ_France_zscore_60d", "LMT_LockheedMartin_ret_1d"], "is_new": true}, {"model_id": "new_h7_STRESS_GradientBoosting_N25_t0", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 7, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWJ_Japan_vol_20d", "EWC_Canada_zscore_60d", "NWL_Newell_ret_20d", "MS_MorganStanley_ret_5d", "LMT_LockheedMartin_vol_20d", "SPY_zscore_60d", "heston_var_ev_h3", "Brent_Oil_FRED_ret_20d", "XLK_Tech_zscore_60d", "EWY_Korea_ret_20d", "EOG_EOGResources_ret_5d", "3M_ret_5d", "XLB_Materials_zscore_60d", "MS_MorganStanley_ret_1d", "Nikkei_Japan_zscore_60d", "PCAR_PaccarInc_ret_5d", "EQR_Equity_ret_1d", "CPB_CampbellSoup_ret_5d", "spx_vol_5d", "gjr_condvar_h1", "INTC_ret_1d", "XLV_Health_zscore_60d", "US1Y_Rate_ret_20d"], "is_new": true}, {"model_id": "new_h7_STRESS_GradientBoosting_N25_t1", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 7, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "HangSeng_HK_ret_1d", "PFE_ret_1d", "IYM_BasicMaterials_ret_20d", "US30Y_Rate_ret_20d", "LMT_LockheedMartin_vol_20d", "3M_vol_20d", "T_ret_1d", "AXP_Amex_vol_20d", "SLB_Schlumberger_ret_5d", "Nikkei_Japan_vol_20d", "BTI_BritishAmerican_ret_20d", "ASX_Australia_vol_20d", "INTC_ret_1d", "PLD_Prologis_ret_5d", "DOW_Price_zscore_60d", "NOC_Northrop_ret_20d", "IWM_SmallCap_vol_20d", "DIS_vol_20d", "SJM_JM_Smucker_ret_5d", "EXC_Exelon_ret_1d", "MS_MorganStanley_zscore_60d", "CCI_CrownCastle_vol_20d", "XLF_Fin_vol_20d"], "is_new": true}, {"model_id": "new_h7_STRESS_GradientBoosting_N25_t2", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 7, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWM_Malaysia_vol_20d", "US5Y_Rate_ret_5d", "Core_PCE_zscore_60d", "AMT_AmericanTower_ret_1d", "LMT_LockheedMartin_vol_20d", "CI_Cigna_vol_20d", "3M_ret_5d", "Nikkei_Japan_vol_20d", "LUV_SouthwestAir_ret_5d", "NOC_Northrop_ret_20d", "SPY_zscore_60d", "TM_Telephone_vol_20d", "MSTR_Bitcoin3_ret_1d", "ORCL_zscore_60d", "EWS_Singapore_ret_5d", "T10Y2Y_Spread_ret_5d", "DHR_ret_1d", "ENB_EnbridgeInc_ret_1d", "MSTR_Bitcoin3_ret_5d", "IYR_US_REIT2_zscore_60d", "LOW_Lowes_ret_20d", "US3M_Rate_vol_20d", "EWH_HongKong_ret_5d"], "is_new": true}, {"model_id": "new_h7_STRESS_GradientBoosting_N25_t3", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 7, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "Core_PCE_zscore_60d", "LOW_Lowes_ret_20d", "EQIX_Equinix_ret_5d", "EWS_Singapore_ret_5d", "AMD_ret_1d", "AMT_AmericanTower_ret_1d", "Industrial_Production_zscore_60d", "CI_Cigna_vol_20d", "NWL_Newell_ret_20d", "NFCI_ret_5d", "LMT_LockheedMartin_vol_20d", "DHR_vol_20d", "EWH_HongKong_ret_5d", "HUM_Humana_ret_5d", "EWM_Malaysia_zscore_60d", "BTI_BritishAmerican_ret_5d", "US3M_Rate_zscore_60d", "GILD_Gilead_ret_20d", "MSTR_Bitcoin3_ret_1d", "heston_var_ev_h7", "PCAR_PaccarInc_ret_5d", "HD_zscore_60d", "SPY_zscore_60d"], "is_new": true}, {"model_id": "new_h7_STRESS_GradientBoosting_N25_t4", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 7, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "MS_MorganStanley_zscore_60d", "ES_Evergy_ret_1d", "GD_GeneralDynamics_zscore_60d", "EWL_Switzerland_zscore_60d", "Core_PCE_zscore_60d", "HUM_Humana_ret_5d", "EWG_Germany_ret_20d", "IYM_BasicMaterials_ret_20d", "ORCL_vol_20d", "AMD_ret_5d", "EWQ_France_zscore_60d", "EWY_Korea_zscore_60d", "Brent_Oil_FRED_ret_5d", "BTI_BritishAmerican_ret_5d", "GILD_Gilead_ret_20d", "Nikkei_Japan_vol_20d", "EQR_Equity_ret_1d", "LMT_LockheedMartin_vol_20d", "US3Y_Rate_ret_5d", "IYR_US_REIT2_zscore_60d", "HD_ret_1d", "CPB_CampbellSoup_vol_20d", "DOW_Price_zscore_60d"], "is_new": true}, {"model_id": "new_h7_STRESS_GradientBoosting_N25_t5", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 7, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "AMD_ret_1d", "Industrial_Production_zscore_60d", "INTC_ret_5d", "EWJ_Japan_vol_20d", "GILD_Gilead_ret_20d", "BLK_BlackRock_zscore_60d", "AORD_AUS_zscore_60d", "VRP_ma5", "HangSeng_HK_vol_20d", "NFCI_ret_5d", "PCAR_PaccarInc_ret_5d", "HD_ret_5d", "TED_Spread_vol_20d", "AMGN_Amgen_ret_1d", "spx_abs_ret_max_5d", "CPB_CampbellSoup_zscore_60d", "CLX_Clorox_vol_20d", "HangSeng_HK_ret_5d", "spx_vol_5d", "gjr_condvar_h1", "MO_AltriaMG_ret_1d", "spx_momentum_3d", "EWL_Switzerland_zscore_60d"], "is_new": true}, {"model_id": "new_h7_STRESS_GradientBoosting_N25_t6", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 7, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "heston_var_ev_h5", "PCAR_PaccarInc_ret_5d", "GE_ret_1d", "Nikkei_Japan_vol_20d", "AMGN_Amgen_ret_1d", "NOC_Northrop_ret_20d", "PG_ret_20d", "SJM_JM_Smucker_ret_5d", "AMZN_ret_5d", "TM_Telephone_vol_20d", "EWA_Australia_ret_1d", "CMCSA_ret_1d", "ES_Evergy_ret_1d", "EXC_Exelon_ret_1d", "ORCL_vol_20d", "LOW_Lowes_ret_20d", "EWC_Canada_zscore_60d", "3M_ret_5d", "SBUX_vol_20d", "EFFR_ret_1d", "EOG_EOGResources_vol_20d", "VOD_Vodafone_zscore_60d", "US3Y_Rate_ret_5d"], "is_new": true}, {"model_id": "new_h7_STRESS_GradientBoosting_N25_t7", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 7, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "INTC_ret_1d", "PAYX_Paychex_vol_20d", "EOG_EOGResources_ret_5d", "EWM_Malaysia_zscore_60d", "CPB_CampbellSoup_ret_5d", "EXC_Exelon_zscore_60d", "US1Y_Rate_ret_5d", "T_ret_1d", "US6M_Rate_ret_20d", "PAYX_Paychex_ret_20d", "gjr_condvar_h1", "Industrial_Production_zscore_60d", "LUV_SouthwestAir_ret_5d", "XOM_ret_1d", "US3M_Rate_zscore_60d", "NOC_Northrop_ret_20d", "SLB_Schlumberger_ret_1d", "hmm_p_stress", "Core_PCE_zscore_60d", "EWA_Australia_ret_1d", "US1Y_Rate_ret_20d", "XLV_Health_zscore_60d", "DE_Deere_ret_5d"], "is_new": true}, {"model_id": "new_h7_STRESS_GradientBoosting_N30_t0", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 7, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "VRP_ma5", "XLK_Tech_zscore_60d", "vix_acceleration_1d", "AMD_ret_5d", "US1Y_Rate_ret_20d", "INTC_ret_1d", "MS_MorganStanley_ret_5d", "XLB_Materials_zscore_60d", "DAX_Germany_zscore_60d", "AXP_Amex_vol_20d", "EFFR_ret_1d", "TGT_Target_zscore_60d", "EWA_Australia_zscore_60d", "JNJ_ret_1d", "DE_Deere_vol_20d", "SLB_Schlumberger_ret_1d", "PAYX_Paychex_zscore_60d", "MSTR_Bitcoin3_ret_1d", "SBUX_ret_5d", "IYM_BasicMaterials_ret_20d", "MSTR_Bitcoin3_ret_20d", "Nikkei_Japan_zscore_60d", "PPL_PPL_ret_1d", "EWL_Switzerland_vol_20d", "CPB_CampbellSoup_ret_5d", "MO_AltriaMG_ret_1d", "TXN_vol_20d", "GD_GeneralDynamics_zscore_60d"], "is_new": true}, {"model_id": "new_h7_STRESS_GradientBoosting_N30_t1", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 7, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "PG_ret_20d", "XLV_Health_zscore_60d", "MSTR_Bitcoin3_ret_20d", "US1Y_Rate_ret_5d", "ITT_ITTInc_ret_5d", "CCI_CrownCastle_vol_20d", "WTI_Oil_FRED_zscore_60d", "VVIX_ret_20d", "3M_vol_20d", "PLD_Prologis_ret_5d", "MS_MorganStanley_zscore_60d", "LOW_Lowes_ret_5d", "Core_PCE_zscore_60d", "MRK_Merck_zscore_60d", "DHR_vol_20d", "AMZN_ret_5d", "EWJ_Japan_vol_20d", "IYM_BasicMaterials_ret_20d", "ASX_Australia_ret_5d", "SLB_Schlumberger_ret_1d", "EWL_Switzerland_zscore_60d", "MS_MorganStanley_ret_5d", "TED_Spread_vol_20d", "NEE_NextEra_ret_20d", "AORD_AUS_zscore_60d", "ORCL_vol_20d", "QQQ_vol_20d", "US7Y_Rate_ret_20d"], "is_new": true}, {"model_id": "new_h7_STRESS_GradientBoosting_N30_t2", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 7, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "CI_Cigna_vol_20d", "DAX_Germany_vol_20d", "EWQ_France_ret_20d", "LUV_SouthwestAir_ret_5d", "TED_Spread_zscore_60d", "gjr_condvar_h1", "heston_ev_h3", "VRP_ma5", "ENB_EnbridgeInc_ret_1d", "SBUX_ret_5d", "3M_vol_20d", "AMT_AmericanTower_ret_1d", "MSTR_Bitcoin3_ret_20d", "VVIX_ret_20d", "US30Y_Rate_ret_20d", "DOW_Price_zscore_60d", "CLX_Clorox_vol_20d", "WTI_Oil_FRED_zscore_60d", "EFFR_ret_1d", "BDX_Becton_Dickinson_ret_20d", "XLK_Tech_zscore_60d", "HD_ret_20d", "INTC_ret_5d", "AORD_AUS_zscore_60d", "ORCL_zscore_60d", "XOM_ret_1d", "EQR_Equity_ret_1d", "HD_zscore_60d"], "is_new": true}, {"model_id": "new_h7_STRESS_GradientBoosting_N30_t3", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 7, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "US5Y_Rate_ret_5d", "VRP_ma5", "TM_Telephone_vol_20d", "CPB_CampbellSoup_zscore_60d", "LLY_zscore_60d", "VOD_Vodafone_zscore_60d", "JNJ_ret_1d", "CPB_CampbellSoup_ret_5d", "heston_var_ev_h3", "PFE_ret_1d", "SBUX_ret_5d", "GE_ret_1d", "EWS_Singapore_ret_5d", "HD_ret_5d", "BA_ret_1d", "DHR_vol_20d", "CCI_CrownCastle_vol_20d", "SCHW_Schwab_ret_5d", "BLK_BlackRock_zscore_60d", "EOG_EOGResources_ret_5d", "AMGN_Amgen_ret_1d", "NOC_Northrop_ret_20d", "EWL_Switzerland_zscore_60d", "GD_GeneralDynamics_zscore_60d", "SPY_zscore_60d", "EWG_Germany_vol_20d", "VVIX_ret_20d", "EWM_Malaysia_ret_1d"], "is_new": true}, {"model_id": "new_h7_STRESS_GradientBoosting_N30_t4", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 7, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "CLX_Clorox_vol_20d", "LUV_SouthwestAir_ret_5d", "EMR_Emerson_ret_20d", "CPB_CampbellSoup_ret_5d", "EWH_HongKong_ret_5d", "US1Y_Rate_ret_5d", "NVDA_vol_20d", "Retail_Sales_zscore_60d", "heston_var_ev_h3", "IBEX_Spain_ret_20d", "CPB_CampbellSoup_ret_20d", "ORCL_zscore_60d", "Core_CPI_zscore_60d", "EWY_Korea_zscore_60d", "HangSeng_HK_ret_5d", "SO_SouthernCo_ret_5d", "LMT_LockheedMartin_vol_20d", "AMD_ret_1d", "PLD_Prologis_ret_5d", "BTI_BritishAmerican_ret_5d", "EWS_Singapore_ret_5d", "US30Y_Rate_ret_20d", "XLY_Disc_vol_20d", "IYR_US_REIT2_zscore_60d", "MS_MorganStanley_ret_5d", "T10Y2Y_Spread_ret_5d", "HD_ret_1d", "LOW_Lowes_ret_5d"], "is_new": true}, {"model_id": "new_h7_STRESS_GradientBoosting_N30_t5", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 7, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EFFR_vol_20d", "HD_ret_1d", "SJM_JM_Smucker_ret_5d", "GILD_Gilead_ret_20d", "EWL_Switzerland_zscore_60d", "US1Y_Rate_ret_20d", "SLB_Schlumberger_ret_5d", "XOM_ret_1d", "EWL_Switzerland_vol_20d", "EWM_Malaysia_zscore_60d", "AVB_AvalonBay_zscore_60d", "EWY_Korea_zscore_60d", "HD_ret_5d", "US3M_Rate_vol_20d", "TXN_vol_20d", "BTI_BritishAmerican_ret_5d", "T10Y2Y_Spread_ret_5d", "INTC_ret_5d", "SPY_zscore_60d", "CTAS_Cintas_vol_20d", "DHR_vol_20d", "EWJ_Japan_vol_20d", "heston_var_ev_h7", "MS_MorganStanley_ret_5d", "PCAR_PaccarInc_ret_5d", "spx_abs_ret_max_5d", "PG_ret_20d", "MSTR_Bitcoin3_ret_5d"], "is_new": true}, {"model_id": "new_h7_STRESS_GradientBoosting_N30_t6", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 7, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "spx_abs_ret_max_5d", "SBUX_vol_20d", "TED_Spread_vol_20d", "ASX_Australia_ret_5d", "HD_zscore_60d", "US3Y_Rate_ret_5d", "US6M_Rate_ret_20d", "ASX_Australia_vol_20d", "EWL_Switzerland_vol_20d", "EWH_HongKong_ret_5d", "EMR_Emerson_ret_20d", "MSTR_Bitcoin3_ret_20d", "EWC_Canada_zscore_60d", "CPB_CampbellSoup_vol_20d", "EQIX_Equinix_ret_5d", "IYR_US_REIT2_zscore_60d", "XLY_Disc_vol_20d", "PLD_Prologis_ret_5d", "ORCL_zscore_60d", "NWL_Newell_ret_20d", "TED_Spread_zscore_60d", "heston_ev_h3", "NEE_NextEra_ret_20d", "QQQ_vol_20d", "Core_PCE_zscore_60d", "EWQ_France_zscore_60d", "AMGN_Amgen_ret_1d", "US1Y_Rate_ret_20d"], "is_new": true}, {"model_id": "new_h7_STRESS_GradientBoosting_N30_t7", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 7, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EXC_Exelon_ret_1d", "EFFR_ret_1d", "GILD_Gilead_ret_20d", "AVB_AvalonBay_zscore_60d", "Core_PCE_zscore_60d", "DIS_vol_20d", "SLB_Schlumberger_ret_1d", "EOG_EOGResources_vol_20d", "SBUX_ret_5d", "EWA_Australia_zscore_60d", "CLX_Clorox_vol_20d", "MO_AltriaMG_ret_1d", "HD_zscore_60d", "DOW_Price_zscore_60d", "gjr_condvar_h1", "DE_Deere_ret_5d", "LOW_Lowes_ret_20d", "SO_SouthernCo_ret_5d", "TGT_Target_zscore_60d", "PLD_Prologis_ret_5d", "BA_ret_1d", "Michigan_Sentiment_ret_20d", "ITT_ITTInc_ret_5d", "PG_ret_20d", "VRP_ma5", "EQR_Equity_ret_1d", "EXC_Exelon_zscore_60d", "GD_GeneralDynamics_zscore_60d"], "is_new": true}, {"model_id": "new_h7_STRESS_RandomForest_N5_t0", "algo": "RandomForest", "regime": "STRESS", "horizon": 7, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "US7Y_Rate_ret_20d", "TED_Spread_zscore_60d", "DAX_Germany_vol_20d"], "is_new": true}, {"model_id": "new_h7_STRESS_RandomForest_N5_t1", "algo": "RandomForest", "regime": "STRESS", "horizon": 7, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "TM_Telephone_ret_1d", "AXP_Amex_vol_20d", "hmm_p_stress"], "is_new": true}, {"model_id": "new_h7_STRESS_RandomForest_N5_t2", "algo": "RandomForest", "regime": "STRESS", "horizon": 7, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "CMCSA_ret_1d", "Nikkei_Japan_zscore_60d", "SJM_JM_Smucker_ret_5d"], "is_new": true}, {"model_id": "new_h7_STRESS_RandomForest_N5_t3", "algo": "RandomForest", "regime": "STRESS", "horizon": 7, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "US3Y_Rate_ret_5d", "EWS_Singapore_ret_5d", "IYR_US_REIT2_zscore_60d"], "is_new": true}, {"model_id": "new_h7_STRESS_RandomForest_N5_t4", "algo": "RandomForest", "regime": "STRESS", "horizon": 7, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWM_Malaysia_ret_1d", "PAYX_Paychex_vol_20d", "DOW_Price_zscore_60d"], "is_new": true}, {"model_id": "new_h7_STRESS_RandomForest_N5_t5", "algo": "RandomForest", "regime": "STRESS", "horizon": 7, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "PAYX_Paychex_vol_20d", "FedFunds_zscore_60d", "MS_MorganStanley_ret_1d"], "is_new": true}, {"model_id": "new_h7_STRESS_RandomForest_N5_t6", "algo": "RandomForest", "regime": "STRESS", "horizon": 7, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "Nikkei_Japan_zscore_60d", "EWC_Canada_zscore_60d", "spx_abs_ret_max_5d"], "is_new": true}, {"model_id": "new_h7_STRESS_RandomForest_N5_t7", "algo": "RandomForest", "regime": "STRESS", "horizon": 7, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EMR_Emerson_ret_20d", "EWQ_France_ret_20d", "HD_zscore_60d"], "is_new": true}, {"model_id": "new_h7_STRESS_RandomForest_N8_t0", "algo": "RandomForest", "regime": "STRESS", "horizon": 7, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "NFCI_ret_5d", "heston_var_ev_h7", "ORCL_zscore_60d", "DE_Deere_vol_20d", "spx_momentum_3d", "Nikkei_Japan_vol_20d"], "is_new": true}, {"model_id": "new_h7_STRESS_RandomForest_N8_t1", "algo": "RandomForest", "regime": "STRESS", "horizon": 7, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "TED_Spread_vol_20d", "Core_CPI_zscore_60d", "HD_zscore_60d", "INTC_ret_1d", "XLY_Disc_vol_20d", "AMD_ret_5d"], "is_new": true}, {"model_id": "new_h7_STRESS_RandomForest_N8_t2", "algo": "RandomForest", "regime": "STRESS", "horizon": 7, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "gjr_condvar_h1", "TED_Spread_vol_20d", "DHR_vol_20d", "HD_zscore_60d", "US1Y_Rate_ret_20d", "CCI_CrownCastle_vol_20d"], "is_new": true}, {"model_id": "new_h7_STRESS_RandomForest_N8_t3", "algo": "RandomForest", "regime": "STRESS", "horizon": 7, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "Brent_Oil_FRED_ret_20d", "Core_PCE_zscore_60d", "MS_MorganStanley_zscore_60d", "FedFunds_zscore_60d", "US3M_Rate_vol_20d", "NFCI_ret_5d"], "is_new": true}, {"model_id": "new_h7_STRESS_RandomForest_N8_t4", "algo": "RandomForest", "regime": "STRESS", "horizon": 7, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "WTI_Oil_FRED_zscore_60d", "NOC_Northrop_ret_20d", "HangSeng_HK_ret_5d", "US5Y_Rate_ret_5d", "MSTR_Bitcoin3_ret_20d", "ITT_ITTInc_ret_5d"], "is_new": true}, {"model_id": "new_h7_STRESS_RandomForest_N8_t5", "algo": "RandomForest", "regime": "STRESS", "horizon": 7, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "AMT_AmericanTower_ret_1d", "hmm_p_stress", "vix_mean_abs_ret_5d", "SLB_Schlumberger_ret_1d", "AMZN_ret_5d", "SBUX_zscore_60d"], "is_new": true}, {"model_id": "new_h7_STRESS_RandomForest_N8_t6", "algo": "RandomForest", "regime": "STRESS", "horizon": 7, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "IYM_BasicMaterials_ret_20d", "spx_abs_ret_max_5d", "EWL_Switzerland_vol_20d", "GILD_Gilead_ret_20d", "TED_Spread_zscore_60d", "HD_ret_20d"], "is_new": true}, {"model_id": "new_h7_STRESS_RandomForest_N8_t7", "algo": "RandomForest", "regime": "STRESS", "horizon": 7, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "TED_Spread_vol_20d", "EWM_Malaysia_ret_1d", "VOD_Vodafone_zscore_60d", "AMZN_ret_5d", "XLV_Health_zscore_60d", "3M_ret_5d"], "is_new": true}, {"model_id": "new_h7_STRESS_RandomForest_N10_t0", "algo": "RandomForest", "regime": "STRESS", "horizon": 7, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "PAYX_Paychex_zscore_60d", "ASX_Australia_ret_5d", "DAX_Germany_zscore_60d", "WTI_Oil_FRED_zscore_60d", "AMZN_ret_5d", "EQIX_Equinix_ret_5d", "SO_SouthernCo_ret_5d", "ASX_Australia_vol_20d"], "is_new": true}, {"model_id": "new_h7_STRESS_RandomForest_N10_t1", "algo": "RandomForest", "regime": "STRESS", "horizon": 7, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "DE_Deere_vol_20d", "Retail_Sales_zscore_60d", "T10Y2Y_Spread_ret_5d", "AMD_ret_5d", "XLF_Fin_vol_20d", "Core_CPI_zscore_60d", "CLX_Clorox_vol_20d", "US3M_Rate_zscore_60d"], "is_new": true}, {"model_id": "new_h7_STRESS_RandomForest_N10_t2", "algo": "RandomForest", "regime": "STRESS", "horizon": 7, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "SBUX_zscore_60d", "EWG_Germany_vol_20d", "LMT_LockheedMartin_ret_1d", "NWL_Newell_ret_20d", "SJM_JM_Smucker_ret_1d", "Brent_Oil_FRED_ret_20d", "XOM_ret_1d", "HUM_Humana_ret_5d"], "is_new": true}, {"model_id": "new_h7_STRESS_RandomForest_N10_t3", "algo": "RandomForest", "regime": "STRESS", "horizon": 7, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "T_ret_1d", "EQR_Equity_ret_1d", "CCI_CrownCastle_vol_20d", "3M_ret_5d", "NFCI_ret_5d", "XLF_Fin_vol_20d", "BA_ret_1d", "TM_Telephone_ret_1d"], "is_new": true}, {"model_id": "new_h7_STRESS_RandomForest_N10_t4", "algo": "RandomForest", "regime": "STRESS", "horizon": 7, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "TXN_vol_20d", "SCHW_Schwab_ret_5d", "INTC_ret_5d", "EWH_HongKong_ret_5d", "LOW_Lowes_ret_5d", "US3Y_Rate_ret_5d", "MSTR_Bitcoin3_ret_1d", "EFFR_ret_1d"], "is_new": true}, {"model_id": "new_h7_STRESS_RandomForest_N10_t5", "algo": "RandomForest", "regime": "STRESS", "horizon": 7, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "PAYX_Paychex_vol_20d", "XLK_Tech_zscore_60d", "LOW_Lowes_ret_20d", "EWL_Switzerland_zscore_60d", "TM_Telephone_vol_20d", "EWG_Germany_vol_20d", "ORCL_zscore_60d", "SJM_JM_Smucker_ret_1d"], "is_new": true}, {"model_id": "new_h7_STRESS_RandomForest_N10_t6", "algo": "RandomForest", "regime": "STRESS", "horizon": 7, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "HD_ret_5d", "ASX_Australia_vol_20d", "IWM_SmallCap_vol_20d", "PAYX_Paychex_vol_20d", "US7Y_Rate_ret_20d", "EWA_Australia_ret_1d", "PFE_ret_1d", "MSTR_Bitcoin3_ret_5d"], "is_new": true}, {"model_id": "new_h7_STRESS_RandomForest_N10_t7", "algo": "RandomForest", "regime": "STRESS", "horizon": 7, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "DHR_vol_20d", "VOD_Vodafone_zscore_60d", "SLB_Schlumberger_ret_5d", "AMT_AmericanTower_ret_1d", "AVB_AvalonBay_zscore_60d", "DAX_Germany_zscore_60d", "EXC_Exelon_ret_1d", "MS_MorganStanley_ret_5d"], "is_new": true}, {"model_id": "new_h7_STRESS_RandomForest_N12_t0", "algo": "RandomForest", "regime": "STRESS", "horizon": 7, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "DAX_Germany_zscore_60d", "Industrial_Production_zscore_60d", "US30Y_Rate_ret_20d", "3M_vol_20d", "EXC_Exelon_ret_1d", "vix_acceleration_1d", "SLB_Schlumberger_ret_5d", "FedFunds_zscore_60d", "US3M_Rate_zscore_60d", "LOW_Lowes_ret_5d"], "is_new": true}, {"model_id": "new_h7_STRESS_RandomForest_N12_t1", "algo": "RandomForest", "regime": "STRESS", "horizon": 7, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "LOW_Lowes_ret_5d", "LUV_SouthwestAir_ret_5d", "US7Y_Rate_ret_20d", "MS_MorganStanley_ret_5d", "SBUX_vol_20d", "DE_Deere_vol_20d", "SPY_zscore_60d", "Retail_Sales_zscore_60d", "TGT_Target_zscore_60d", "EXC_Exelon_zscore_60d"], "is_new": true}, {"model_id": "new_h7_STRESS_RandomForest_N12_t2", "algo": "RandomForest", "regime": "STRESS", "horizon": 7, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "TED_Spread_zscore_60d", "SO_SouthernCo_ret_5d", "Michigan_Sentiment_ret_20d", "EWL_Switzerland_vol_20d", "EOG_EOGResources_vol_20d", "CPB_CampbellSoup_zscore_60d", "MS_MorganStanley_ret_1d", "ENB_EnbridgeInc_ret_1d", "EWJ_Japan_vol_20d", "US7Y_Rate_ret_20d"], "is_new": true}, {"model_id": "new_h7_STRESS_RandomForest_N12_t3", "algo": "RandomForest", "regime": "STRESS", "horizon": 7, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "ES_Evergy_ret_1d", "EWY_Korea_zscore_60d", "EXC_Exelon_ret_1d", "EWA_Australia_ret_1d", "SLB_Schlumberger_ret_5d", "BA_ret_1d", "PAYX_Paychex_ret_20d", "HD_ret_20d", "EXC_Exelon_zscore_60d", "EQR_Equity_ret_1d"], "is_new": true}, {"model_id": "new_h7_STRESS_RandomForest_N12_t4", "algo": "RandomForest", "regime": "STRESS", "horizon": 7, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "HangSeng_HK_vol_20d", "PAYX_Paychex_zscore_60d", "XLB_Materials_zscore_60d", "EXC_Exelon_zscore_60d", "PLD_Prologis_ret_5d", "vix_mean_abs_ret_5d", "NEE_NextEra_ret_20d", "HD_ret_20d", "hmm_p_stress", "CPB_CampbellSoup_ret_5d"], "is_new": true}, {"model_id": "new_h7_STRESS_RandomForest_N12_t5", "algo": "RandomForest", "regime": "STRESS", "horizon": 7, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "T10Y2Y_Spread_ret_5d", "ITT_ITTInc_ret_5d", "EWC_Canada_zscore_60d", "DHR_vol_20d", "ORCL_vol_20d", "MSTR_Bitcoin3_ret_5d", "DE_Deere_ret_5d", "LMT_LockheedMartin_ret_1d", "EOG_EOGResources_vol_20d", "US1Y_Rate_ret_5d"], "is_new": true}, {"model_id": "new_h7_STRESS_RandomForest_N12_t6", "algo": "RandomForest", "regime": "STRESS", "horizon": 7, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EXC_Exelon_zscore_60d", "EXC_Exelon_ret_1d", "ENB_EnbridgeInc_ret_1d", "hmm_p_stress", "heston_ev_h3", "WTI_Oil_FRED_zscore_60d", "US1Y_Rate_ret_5d", "SBUX_ret_5d", "US3Y_Rate_ret_5d", "NOC_Northrop_ret_20d"], "is_new": true}, {"model_id": "new_h7_STRESS_RandomForest_N12_t7", "algo": "RandomForest", "regime": "STRESS", "horizon": 7, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "MSTR_Bitcoin3_ret_5d", "BTI_BritishAmerican_ret_20d", "CPB_CampbellSoup_vol_20d", "EWC_Canada_zscore_60d", "TED_Spread_zscore_60d", "EXC_Exelon_zscore_60d", "INTC_ret_1d", "CLX_Clorox_vol_20d", "EWA_Australia_ret_1d", "3M_vol_20d"], "is_new": true}, {"model_id": "new_h7_STRESS_RandomForest_N15_t0", "algo": "RandomForest", "regime": "STRESS", "horizon": 7, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "AMZN_ret_5d", "EWG_Germany_vol_20d", "vix_acceleration_1d", "Retail_Sales_zscore_60d", "DE_Deere_vol_20d", "XLF_Fin_vol_20d", "LUV_SouthwestAir_ret_5d", "NOC_Northrop_ret_20d", "EFFR_vol_20d", "LLY_zscore_60d", "AORD_AUS_zscore_60d", "LMT_LockheedMartin_ret_1d", "SJM_JM_Smucker_ret_1d"], "is_new": true}, {"model_id": "new_h7_STRESS_RandomForest_N15_t1", "algo": "RandomForest", "regime": "STRESS", "horizon": 7, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "US3M_Rate_zscore_60d", "Retail_Sales_zscore_60d", "EXC_Exelon_zscore_60d", "DE_Deere_vol_20d", "BTI_BritishAmerican_ret_20d", "AXP_Amex_vol_20d", "NWL_Newell_ret_20d", "spx_vol_5d", "Nikkei_Japan_vol_20d", "HD_ret_1d", "MS_MorganStanley_ret_1d", "PLD_Prologis_ret_5d", "EWM_Malaysia_zscore_60d"], "is_new": true}, {"model_id": "new_h7_STRESS_RandomForest_N15_t2", "algo": "RandomForest", "regime": "STRESS", "horizon": 7, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "MS_MorganStanley_ret_1d", "AMGN_Amgen_ret_1d", "Nikkei_Japan_zscore_60d", "XLV_Health_zscore_60d", "HD_zscore_60d", "DE_Deere_vol_20d", "BLK_BlackRock_zscore_60d", "DOW_Price_zscore_60d", "EQR_Equity_ret_1d", "hmm_p_stress", "LLY_zscore_60d", "TM_Telephone_ret_1d", "TED_Spread_vol_20d"], "is_new": true}, {"model_id": "new_h7_STRESS_RandomForest_N15_t3", "algo": "RandomForest", "regime": "STRESS", "horizon": 7, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "DAX_Germany_vol_20d", "MSTR_Bitcoin3_ret_1d", "HD_ret_5d", "US3M_Rate_vol_20d", "PAYX_Paychex_vol_20d", "US5Y_Rate_ret_5d", "US1Y_Rate_ret_20d", "hmm_p_stress", "HangSeng_HK_vol_20d", "Retail_Sales_zscore_60d", "heston_ev_h3", "EXC_Exelon_ret_1d", "AORD_AUS_zscore_60d"], "is_new": true}, {"model_id": "new_h7_STRESS_RandomForest_N15_t4", "algo": "RandomForest", "regime": "STRESS", "horizon": 7, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "T10Y2Y_Spread_ret_5d", "Core_CPI_zscore_60d", "EWM_Malaysia_vol_20d", "AMGN_Amgen_ret_1d", "BTI_BritishAmerican_ret_5d", "3M_vol_20d", "EWM_Malaysia_zscore_60d", "IWM_SmallCap_vol_20d", "MS_MorganStanley_ret_1d", "CI_Cigna_vol_20d", "FedFunds_zscore_60d", "WTI_Oil_FRED_zscore_60d", "XOM_ret_20d"], "is_new": true}, {"model_id": "new_h7_STRESS_RandomForest_N15_t5", "algo": "RandomForest", "regime": "STRESS", "horizon": 7, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "M_Macys_vol_20d", "Core_PCE_zscore_60d", "DAX_Germany_vol_20d", "hmm_p_stress", "NWL_Newell_ret_20d", "PFE_ret_1d", "SBUX_vol_20d", "EXC_Exelon_zscore_60d", "PAYX_Paychex_vol_20d", "PLD_Prologis_ret_5d", "EWH_HongKong_ret_5d", "Brent_Oil_FRED_ret_20d", "EWL_Switzerland_zscore_60d"], "is_new": true}, {"model_id": "new_h7_STRESS_RandomForest_N15_t6", "algo": "RandomForest", "regime": "STRESS", "horizon": 7, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "CPB_CampbellSoup_vol_20d", "SBUX_ret_5d", "AXP_Amex_ret_20d", "DIS_vol_20d", "PLD_Prologis_ret_5d", "DE_Deere_vol_20d", "US3M_Rate_vol_20d", "EWY_Korea_ret_20d", "IYR_US_REIT2_zscore_60d", "LMT_LockheedMartin_ret_1d", "US3Y_Rate_ret_5d", "QQQ_vol_20d", "EWS_Singapore_ret_5d"], "is_new": true}, {"model_id": "new_h7_STRESS_RandomForest_N15_t7", "algo": "RandomForest", "regime": "STRESS", "horizon": 7, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "PPL_PPL_ret_1d", "spx_abs_ret_max_5d", "SJM_JM_Smucker_ret_1d", "INTC_ret_1d", "BDX_Becton_Dickinson_ret_20d", "ES_Evergy_ret_1d", "GD_GeneralDynamics_zscore_60d", "SLB_Schlumberger_ret_1d", "EWA_Australia_zscore_60d", "WTI_Oil_FRED_zscore_60d", "VOD_Vodafone_zscore_60d", "Core_CPI_zscore_60d", "DHR_vol_20d"], "is_new": true}, {"model_id": "new_h7_STRESS_RandomForest_N20_t0", "algo": "RandomForest", "regime": "STRESS", "horizon": 7, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "Industrial_Production_zscore_60d", "XLF_Fin_vol_20d", "SPY_zscore_60d", "US7Y_Rate_ret_20d", "ES_Evergy_ret_1d", "TED_Spread_vol_20d", "EWH_HongKong_ret_5d", "PCAR_PaccarInc_ret_5d", "PLD_Prologis_ret_5d", "LMT_LockheedMartin_vol_20d", "EQR_Equity_ret_1d", "DOW_Price_zscore_60d", "EWA_Australia_zscore_60d", "LMT_LockheedMartin_ret_1d", "MSTR_Bitcoin3_ret_1d", "LOW_Lowes_ret_20d", "LLY_zscore_60d", "CCI_CrownCastle_vol_20d"], "is_new": true}, {"model_id": "new_h7_STRESS_RandomForest_N20_t1", "algo": "RandomForest", "regime": "STRESS", "horizon": 7, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "SCHW_Schwab_ret_5d", "heston_var_ev_h5", "IYM_BasicMaterials_ret_20d", "HD_ret_1d", "CPB_CampbellSoup_ret_20d", "3M_ret_5d", "NVDA_vol_20d", "SBUX_vol_20d", "LOW_Lowes_ret_20d", "SO_SouthernCo_ret_5d", "LMT_LockheedMartin_vol_20d", "SPY_zscore_60d", "TED_Spread_zscore_60d", "EWM_Malaysia_vol_20d", "GILD_Gilead_ret_20d", "MSTR_Bitcoin3_ret_20d", "DIS_vol_20d", "IWM_SmallCap_vol_20d"], "is_new": true}, {"model_id": "new_h7_STRESS_RandomForest_N20_t2", "algo": "RandomForest", "regime": "STRESS", "horizon": 7, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "US7Y_Rate_ret_20d", "Brent_Oil_FRED_ret_20d", "BTI_BritishAmerican_ret_5d", "CTAS_Cintas_vol_20d", "M_Macys_vol_20d", "SLB_Schlumberger_ret_1d", "US1Y_Rate_ret_5d", "3M_ret_5d", "EQR_Equity_ret_1d", "SJM_JM_Smucker_ret_5d", "Core_CPI_zscore_60d", "EQIX_Equinix_ret_5d", "IYM_BasicMaterials_ret_20d", "LMT_LockheedMartin_vol_20d", "GD_GeneralDynamics_zscore_60d", "EOG_EOGResources_ret_5d", "LOW_Lowes_ret_5d", "AXP_Amex_vol_20d"], "is_new": true}, {"model_id": "new_h7_STRESS_RandomForest_N20_t3", "algo": "RandomForest", "regime": "STRESS", "horizon": 7, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "DE_Deere_vol_20d", "NWL_Newell_ret_20d", "vix_mean_abs_ret_5d", "DAX_Germany_zscore_60d", "AMT_AmericanTower_ret_1d", "GE_ret_1d", "SCHW_Schwab_ret_5d", "CMCSA_ret_1d", "HD_ret_5d", "TM_Telephone_ret_1d", "US3M_Rate_vol_20d", "ASX_Australia_ret_5d", "SLB_Schlumberger_ret_5d", "AXP_Amex_ret_20d", "EXC_Exelon_ret_1d", "US7Y_Rate_ret_20d", "HangSeng_HK_ret_5d", "PCAR_PaccarInc_ret_5d"], "is_new": true}, {"model_id": "new_h7_STRESS_RandomForest_N20_t4", "algo": "RandomForest", "regime": "STRESS", "horizon": 7, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWM_Malaysia_zscore_60d", "EWQ_France_ret_20d", "BA_ret_1d", "MSTR_Bitcoin3_ret_1d", "MS_MorganStanley_ret_5d", "NWL_Newell_ret_20d", "Michigan_Sentiment_ret_20d", "AVB_AvalonBay_zscore_60d", "GILD_Gilead_ret_20d", "Industrial_Production_zscore_60d", "DOW_Price_zscore_60d", "NVDA_vol_20d", "US5Y_Rate_ret_5d", "DE_Deere_vol_20d", "EWL_Switzerland_vol_20d", "TGT_Target_zscore_60d", "EMR_Emerson_ret_20d", "ENB_EnbridgeInc_ret_1d"], "is_new": true}, {"model_id": "new_h7_STRESS_RandomForest_N20_t5", "algo": "RandomForest", "regime": "STRESS", "horizon": 7, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "LMT_LockheedMartin_vol_20d", "TM_Telephone_ret_1d", "BLK_BlackRock_zscore_60d", "DOW_Price_zscore_60d", "US1Y_Rate_ret_5d", "XOM_ret_1d", "FedFunds_zscore_60d", "Nikkei_Japan_vol_20d", "T_ret_1d", "EWA_Australia_zscore_60d", "AORD_AUS_zscore_60d", "IYR_US_REIT2_zscore_60d", "EWM_Malaysia_zscore_60d", "QQQ_vol_20d", "TGT_Target_zscore_60d", "XLY_Disc_vol_20d", "IBEX_Spain_ret_20d", "AMGN_Amgen_ret_1d"], "is_new": true}, {"model_id": "new_h7_STRESS_RandomForest_N20_t6", "algo": "RandomForest", "regime": "STRESS", "horizon": 7, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "XLB_Materials_zscore_60d", "TGT_Target_zscore_60d", "MS_MorganStanley_ret_1d", "GILD_Gilead_ret_20d", "T10Y2Y_Spread_ret_5d", "Nikkei_Japan_vol_20d", "HD_ret_1d", "spx_vol_5d", "Nikkei_Japan_zscore_60d", "ORCL_vol_20d", "INTC_ret_1d", "US3M_Rate_zscore_60d", "EWG_Germany_ret_20d", "LUV_SouthwestAir_ret_5d", "EMR_Emerson_ret_20d", "EWY_Korea_ret_20d", "SBUX_ret_5d", "CPB_CampbellSoup_ret_5d"], "is_new": true}, {"model_id": "new_h7_STRESS_RandomForest_N20_t7", "algo": "RandomForest", "regime": "STRESS", "horizon": 7, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "vix_acceleration_1d", "SBUX_vol_20d", "3M_vol_20d", "Core_PCE_zscore_60d", "CPB_CampbellSoup_ret_5d", "US3Y_Rate_ret_5d", "DHR_ret_1d", "heston_var_ev_h7", "heston_var_ev_h3", "EWM_Malaysia_ret_1d", "Industrial_Production_zscore_60d", "PAYX_Paychex_ret_20d", "EWQ_France_ret_20d", "spx_momentum_3d", "EWH_HongKong_ret_5d", "VVIX_ret_20d", "EWM_Malaysia_zscore_60d", "ES_Evergy_ret_1d"], "is_new": true}, {"model_id": "new_h7_STRESS_RandomForest_N25_t0", "algo": "RandomForest", "regime": "STRESS", "horizon": 7, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EFFR_vol_20d", "QQQ_vol_20d", "spx_momentum_3d", "CPB_CampbellSoup_ret_20d", "US5Y_Rate_ret_5d", "Nikkei_Japan_vol_20d", "PPL_PPL_ret_1d", "AMD_ret_5d", "EWM_Malaysia_ret_1d", "Core_PCE_zscore_60d", "heston_var_ev_h5", "IWM_SmallCap_vol_20d", "EWG_Germany_ret_20d", "SCHW_Schwab_ret_5d", "heston_ev_h3", "TED_Spread_zscore_60d", "ASX_Australia_ret_5d", "VRP_ma5", "XLK_Tech_zscore_60d", "DIS_vol_20d", "AMT_AmericanTower_ret_1d", "INTC_ret_1d", "EWY_Korea_ret_20d"], "is_new": true}, {"model_id": "new_h7_STRESS_RandomForest_N25_t1", "algo": "RandomForest", "regime": "STRESS", "horizon": 7, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "IYR_US_REIT2_zscore_60d", "EWL_Switzerland_zscore_60d", "Brent_Oil_FRED_ret_5d", "EFFR_ret_1d", "TED_Spread_zscore_60d", "GILD_Gilead_ret_20d", "PAYX_Paychex_vol_20d", "VVIX_ret_20d", "EWY_Korea_zscore_60d", "EWS_Singapore_ret_5d", "vix_acceleration_1d", "CI_Cigna_vol_20d", "LUV_SouthwestAir_ret_5d", "DHR_vol_20d", "CCI_CrownCastle_vol_20d", "Michigan_Sentiment_ret_20d", "ORCL_vol_20d", "EXC_Exelon_ret_1d", "heston_var_ev_h3", "HD_ret_5d", "Industrial_Production_zscore_60d", "AMD_ret_1d", "PLD_Prologis_ret_5d"], "is_new": true}, {"model_id": "new_h7_STRESS_RandomForest_N25_t2", "algo": "RandomForest", "regime": "STRESS", "horizon": 7, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "Retail_Sales_zscore_60d", "SPY_zscore_60d", "SJM_JM_Smucker_ret_1d", "vix_acceleration_1d", "EWM_Malaysia_vol_20d", "AORD_AUS_zscore_60d", "AXP_Amex_vol_20d", "LLY_zscore_60d", "PPL_PPL_ret_1d", "LOW_Lowes_ret_20d", "EOG_EOGResources_ret_5d", "IYR_US_REIT2_zscore_60d", "spx_vol_5d", "MO_AltriaMG_ret_1d", "SO_SouthernCo_ret_5d", "EWA_Australia_ret_1d", "MSTR_Bitcoin3_ret_20d", "HD_ret_20d", "DE_Deere_vol_20d", "US1Y_Rate_ret_5d", "MS_MorganStanley_zscore_60d", "Michigan_Sentiment_ret_20d", "VVIX_ret_20d"], "is_new": true}, {"model_id": "new_h7_STRESS_RandomForest_N25_t3", "algo": "RandomForest", "regime": "STRESS", "horizon": 7, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "AMD_ret_1d", "AMGN_Amgen_ret_1d", "SBUX_ret_5d", "ENB_EnbridgeInc_ret_1d", "CPB_CampbellSoup_ret_20d", "US30Y_Rate_ret_20d", "EOG_EOGResources_ret_5d", "MS_MorganStanley_zscore_60d", "LLY_zscore_60d", "EWY_Korea_ret_20d", "IYR_US_REIT2_zscore_60d", "EWM_Malaysia_vol_20d", "TM_Telephone_vol_20d", "EMR_Emerson_ret_20d", "Core_CPI_zscore_60d", "SPY_zscore_60d", "CPB_CampbellSoup_zscore_60d", "AORD_AUS_zscore_60d", "TXN_vol_20d", "hmm_p_stress", "NEE_NextEra_ret_20d", "HD_ret_5d", "DE_Deere_vol_20d"], "is_new": true}, {"model_id": "new_h7_STRESS_RandomForest_N25_t4", "algo": "RandomForest", "regime": "STRESS", "horizon": 7, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "CCI_CrownCastle_vol_20d", "FedFunds_zscore_60d", "PLD_Prologis_ret_5d", "HD_zscore_60d", "US6M_Rate_ret_20d", "EWM_Malaysia_vol_20d", "IWM_SmallCap_vol_20d", "EXC_Exelon_ret_1d", "US3Y_Rate_ret_5d", "HangSeng_HK_vol_20d", "Core_CPI_zscore_60d", "3M_vol_20d", "XLK_Tech_zscore_60d", "SCHW_Schwab_ret_5d", "MRK_Merck_zscore_60d", "SPY_zscore_60d", "PPL_PPL_ret_1d", "SBUX_vol_20d", "ORCL_vol_20d", "VRP_ma5", "TED_Spread_vol_20d", "SBUX_ret_5d", "heston_var_ev_h3"], "is_new": true}, {"model_id": "new_h7_STRESS_RandomForest_N25_t5", "algo": "RandomForest", "regime": "STRESS", "horizon": 7, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "XLK_Tech_zscore_60d", "VOD_Vodafone_zscore_60d", "XLY_Disc_vol_20d", "EWG_Germany_vol_20d", "LUV_SouthwestAir_ret_5d", "GD_GeneralDynamics_zscore_60d", "EMR_Emerson_ret_20d", "PAYX_Paychex_zscore_60d", "NWL_Newell_ret_20d", "DAX_Germany_zscore_60d", "XLB_Materials_zscore_60d", "EWJ_Japan_vol_20d", "CI_Cigna_vol_20d", "PG_ret_20d", "PPL_PPL_ret_1d", "ASX_Australia_vol_20d", "SBUX_zscore_60d", "PAYX_Paychex_vol_20d", "DE_Deere_ret_5d", "EWQ_France_zscore_60d", "ASX_Australia_ret_5d", "Nikkei_Japan_vol_20d", "Nikkei_Japan_zscore_60d"], "is_new": true}, {"model_id": "new_h7_STRESS_RandomForest_N25_t6", "algo": "RandomForest", "regime": "STRESS", "horizon": 7, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "ES_Evergy_ret_1d", "ASX_Australia_vol_20d", "EWY_Korea_ret_20d", "CTAS_Cintas_vol_20d", "LMT_LockheedMartin_vol_20d", "US6M_Rate_ret_20d", "DAX_Germany_vol_20d", "IYM_BasicMaterials_ret_20d", "PLD_Prologis_ret_5d", "EWC_Canada_zscore_60d", "CPB_CampbellSoup_vol_20d", "IBEX_Spain_ret_20d", "EWA_Australia_ret_1d", "AORD_AUS_zscore_60d", "LOW_Lowes_ret_20d", "XLF_Fin_vol_20d", "MS_MorganStanley_zscore_60d", "Core_CPI_zscore_60d", "MSTR_Bitcoin3_ret_5d", "AMGN_Amgen_ret_1d", "LOW_Lowes_ret_5d", "AMZN_ret_5d", "VRP_ma5"], "is_new": true}, {"model_id": "new_h7_STRESS_RandomForest_N25_t7", "algo": "RandomForest", "regime": "STRESS", "horizon": 7, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "CMCSA_ret_1d", "SPY_zscore_60d", "EWG_Germany_vol_20d", "SLB_Schlumberger_ret_5d", "M_Macys_vol_20d", "AVB_AvalonBay_zscore_60d", "HD_zscore_60d", "AMGN_Amgen_ret_1d", "heston_var_ev_h5", "US30Y_Rate_ret_20d", "CPB_CampbellSoup_zscore_60d", "US3M_Rate_zscore_60d", "US1Y_Rate_ret_5d", "DE_Deere_vol_20d", "ENB_EnbridgeInc_ret_1d", "DHR_vol_20d", "TXN_vol_20d", "EWS_Singapore_ret_5d", "DIS_vol_20d", "XOM_ret_1d", "AMD_ret_5d", "AMD_ret_1d", "Michigan_Sentiment_ret_20d"], "is_new": true}, {"model_id": "new_h7_STRESS_RandomForest_N30_t0", "algo": "RandomForest", "regime": "STRESS", "horizon": 7, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "ENB_EnbridgeInc_ret_1d", "GILD_Gilead_ret_20d", "INTC_ret_5d", "TED_Spread_zscore_60d", "BTI_BritishAmerican_ret_5d", "VVIX_ret_20d", "PAYX_Paychex_zscore_60d", "EFFR_vol_20d", "EWQ_France_ret_20d", "US3M_Rate_vol_20d", "M_Macys_vol_20d", "CPB_CampbellSoup_vol_20d", "DIS_vol_20d", "HD_zscore_60d", "vix_mean_abs_ret_5d", "DAX_Germany_zscore_60d", "EOG_EOGResources_vol_20d", "ORCL_vol_20d", "CPB_CampbellSoup_ret_5d", "gjr_condvar_h1", "ORCL_zscore_60d", "heston_var_ev_h3", "EWS_Singapore_ret_5d", "MS_MorganStanley_zscore_60d", "NFCI_ret_5d", "AXP_Amex_ret_20d", "MO_AltriaMG_ret_1d", "EWA_Australia_zscore_60d"], "is_new": true}, {"model_id": "new_h7_STRESS_RandomForest_N30_t1", "algo": "RandomForest", "regime": "STRESS", "horizon": 7, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "ENB_EnbridgeInc_ret_1d", "HUM_Humana_ret_5d", "XLV_Health_zscore_60d", "spx_abs_ret_max_5d", "AMGN_Amgen_ret_1d", "heston_var_ev_h3", "ITT_ITTInc_ret_5d", "HD_zscore_60d", "BTI_BritishAmerican_ret_5d", "EWL_Switzerland_zscore_60d", "EMR_Emerson_ret_20d", "EWG_Germany_vol_20d", "PAYX_Paychex_zscore_60d", "CMCSA_ret_1d", "LOW_Lowes_ret_20d", "EWQ_France_zscore_60d", "MO_AltriaMG_ret_1d", "PG_ret_20d", "VVIX_ret_20d", "Industrial_Production_zscore_60d", "EWM_Malaysia_vol_20d", "HangSeng_HK_ret_1d", "WTI_Oil_FRED_zscore_60d", "LUV_SouthwestAir_ret_5d", "DAX_Germany_zscore_60d", "T_ret_1d", "CPB_CampbellSoup_zscore_60d", "heston_var_ev_h7"], "is_new": true}, {"model_id": "new_h7_STRESS_RandomForest_N30_t2", "algo": "RandomForest", "regime": "STRESS", "horizon": 7, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "PPL_PPL_ret_1d", "EWQ_France_zscore_60d", "MSTR_Bitcoin3_ret_20d", "LUV_SouthwestAir_ret_5d", "EWA_Australia_ret_1d", "CLX_Clorox_vol_20d", "US3Y_Rate_ret_5d", "EWQ_France_ret_20d", "EWC_Canada_zscore_60d", "M_Macys_vol_20d", "MSTR_Bitcoin3_ret_1d", "Industrial_Production_zscore_60d", "DHR_vol_20d", "3M_ret_5d", "vix_acceleration_1d", "INTC_ret_5d", "GE_ret_1d", "Michigan_Sentiment_ret_20d", "hmm_p_stress", "XOM_ret_1d", "SO_SouthernCo_ret_5d", "AMD_ret_1d", "LOW_Lowes_ret_20d", "HangSeng_HK_vol_20d", "heston_var_ev_h5", "CPB_CampbellSoup_ret_5d", "EWG_Germany_vol_20d", "DE_Deere_vol_20d"], "is_new": true}, {"model_id": "new_h7_STRESS_RandomForest_N30_t3", "algo": "RandomForest", "regime": "STRESS", "horizon": 7, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "AORD_AUS_zscore_60d", "AXP_Amex_vol_20d", "CPB_CampbellSoup_vol_20d", "BLK_BlackRock_zscore_60d", "CTAS_Cintas_vol_20d", "HD_zscore_60d", "JNJ_ret_1d", "HangSeng_HK_vol_20d", "SBUX_ret_5d", "US1Y_Rate_ret_5d", "EWQ_France_ret_20d", "Nikkei_Japan_vol_20d", "GILD_Gilead_ret_20d", "EWL_Switzerland_zscore_60d", "PG_ret_20d", "EXC_Exelon_zscore_60d", "SLB_Schlumberger_ret_5d", "DHR_vol_20d", "Michigan_Sentiment_ret_20d", "DE_Deere_vol_20d", "EWL_Switzerland_vol_20d", "DIS_vol_20d", "US3M_Rate_zscore_60d", "heston_ev_h3", "NEE_NextEra_ret_20d", "CPB_CampbellSoup_ret_20d", "US6M_Rate_ret_20d", "SJM_JM_Smucker_ret_5d"], "is_new": true}, {"model_id": "new_h7_STRESS_RandomForest_N30_t4", "algo": "RandomForest", "regime": "STRESS", "horizon": 7, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "LMT_LockheedMartin_vol_20d", "WTI_Oil_FRED_zscore_60d", "Brent_Oil_FRED_ret_20d", "DHR_vol_20d", "FedFunds_zscore_60d", "ORCL_zscore_60d", "3M_vol_20d", "HD_ret_1d", "EWA_Australia_zscore_60d", "CPB_CampbellSoup_ret_20d", "DIS_vol_20d", "EXC_Exelon_ret_1d", "ASX_Australia_ret_5d", "DE_Deere_ret_5d", "spx_vol_5d", "HD_ret_5d", "BDX_Becton_Dickinson_ret_20d", "US7Y_Rate_ret_20d", "heston_ev_h3", "HD_ret_20d", "VOD_Vodafone_zscore_60d", "EFFR_ret_1d", "CMCSA_ret_1d", "spx_abs_ret_max_5d", "XLB_Materials_zscore_60d", "hmm_p_stress", "CPB_CampbellSoup_vol_20d", "AMT_AmericanTower_ret_1d"], "is_new": true}, {"model_id": "new_h7_STRESS_RandomForest_N30_t5", "algo": "RandomForest", "regime": "STRESS", "horizon": 7, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "HD_ret_20d", "BA_ret_1d", "ENB_EnbridgeInc_ret_1d", "EFFR_ret_1d", "GE_ret_1d", "TXN_vol_20d", "BLK_BlackRock_zscore_60d", "NOC_Northrop_ret_20d", "EWY_Korea_zscore_60d", "EOG_EOGResources_vol_20d", "AMD_ret_1d", "IBEX_Spain_ret_20d", "US1Y_Rate_ret_5d", "US1Y_Rate_ret_20d", "Nikkei_Japan_vol_20d", "EWA_Australia_zscore_60d", "EWH_HongKong_ret_5d", "EOG_EOGResources_ret_5d", "AMT_AmericanTower_ret_1d", "heston_var_ev_h7", "ES_Evergy_ret_1d", "ORCL_vol_20d", "SO_SouthernCo_ret_5d", "EWL_Switzerland_vol_20d", "PG_ret_20d", "BDX_Becton_Dickinson_ret_20d", "AORD_AUS_zscore_60d", "PFE_ret_1d"], "is_new": true}, {"model_id": "new_h7_STRESS_RandomForest_N30_t6", "algo": "RandomForest", "regime": "STRESS", "horizon": 7, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "TM_Telephone_ret_1d", "spx_momentum_3d", "Core_CPI_zscore_60d", "AMD_ret_1d", "spx_vol_5d", "EWS_Singapore_ret_5d", "NWL_Newell_ret_20d", "CLX_Clorox_vol_20d", "CI_Cigna_vol_20d", "LMT_LockheedMartin_ret_1d", "AVB_AvalonBay_zscore_60d", "Michigan_Sentiment_ret_20d", "IBEX_Spain_ret_20d", "PAYX_Paychex_ret_20d", "LMT_LockheedMartin_vol_20d", "MSTR_Bitcoin3_ret_20d", "MS_MorganStanley_zscore_60d", "CPB_CampbellSoup_ret_20d", "EWA_Australia_zscore_60d", "NOC_Northrop_ret_20d", "Brent_Oil_FRED_ret_20d", "CCI_CrownCastle_vol_20d", "BA_ret_1d", "Retail_Sales_zscore_60d", "XLF_Fin_vol_20d", "FedFunds_zscore_60d", "PCAR_PaccarInc_ret_5d", "EWM_Malaysia_ret_1d"], "is_new": true}, {"model_id": "new_h7_STRESS_RandomForest_N30_t7", "algo": "RandomForest", "regime": "STRESS", "horizon": 7, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "CPB_CampbellSoup_ret_5d", "TED_Spread_vol_20d", "EXC_Exelon_zscore_60d", "ASX_Australia_vol_20d", "M_Macys_vol_20d", "CPB_CampbellSoup_ret_20d", "GILD_Gilead_ret_20d", "TGT_Target_zscore_60d", "INTC_ret_5d", "GE_ret_1d", "ORCL_vol_20d", "ASX_Australia_ret_5d", "AXP_Amex_vol_20d", "SO_SouthernCo_ret_5d", "vix_mean_abs_ret_5d", "BTI_BritishAmerican_ret_5d", "SLB_Schlumberger_ret_1d", "HD_zscore_60d", "spx_abs_ret_max_5d", "heston_var_ev_h3", "3M_vol_20d", "EWG_Germany_vol_20d", "DE_Deere_ret_5d", "AMZN_ret_5d", "PAYX_Paychex_zscore_60d", "SBUX_vol_20d", "EQIX_Equinix_ret_5d", "Michigan_Sentiment_ret_20d"], "is_new": true}, {"model_id": "new_h7_STRESS_LogisticRegression_N5_t0", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 7, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "TXN_vol_20d", "GD_GeneralDynamics_zscore_60d", "XLV_Health_zscore_60d"], "is_new": true}, {"model_id": "new_h7_STRESS_LogisticRegression_N5_t1", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 7, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "PAYX_Paychex_ret_20d", "Michigan_Sentiment_ret_20d", "TED_Spread_vol_20d"], "is_new": true}, {"model_id": "new_h7_STRESS_LogisticRegression_N5_t2", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 7, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "TXN_vol_20d", "EWM_Malaysia_vol_20d", "HD_zscore_60d"], "is_new": true}, {"model_id": "new_h7_STRESS_LogisticRegression_N5_t3", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 7, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "Brent_Oil_FRED_ret_20d", "TED_Spread_vol_20d", "US1Y_Rate_ret_20d"], "is_new": true}, {"model_id": "new_h7_STRESS_LogisticRegression_N5_t4", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 7, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EMR_Emerson_ret_20d", "LUV_SouthwestAir_ret_5d", "EFFR_ret_1d"], "is_new": true}, {"model_id": "new_h7_STRESS_LogisticRegression_N5_t5", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 7, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "DOW_Price_zscore_60d", "US3M_Rate_vol_20d", "MRK_Merck_zscore_60d"], "is_new": true}, {"model_id": "new_h7_STRESS_LogisticRegression_N5_t6", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 7, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "ASX_Australia_vol_20d", "EWQ_France_zscore_60d", "MRK_Merck_zscore_60d"], "is_new": true}, {"model_id": "new_h7_STRESS_LogisticRegression_N5_t7", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 7, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "heston_var_ev_h3", "EWC_Canada_zscore_60d", "US5Y_Rate_ret_5d"], "is_new": true}, {"model_id": "new_h7_STRESS_LogisticRegression_N8_t0", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 7, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "ES_Evergy_ret_1d", "SO_SouthernCo_ret_5d", "EFFR_vol_20d", "IWM_SmallCap_vol_20d", "EOG_EOGResources_ret_5d", "HD_ret_5d"], "is_new": true}, {"model_id": "new_h7_STRESS_LogisticRegression_N8_t1", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 7, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "vix_acceleration_1d", "CMCSA_ret_1d", "hmm_p_stress", "CLX_Clorox_vol_20d", "EWH_HongKong_ret_5d", "ES_Evergy_ret_1d"], "is_new": true}, {"model_id": "new_h7_STRESS_LogisticRegression_N8_t2", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 7, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "DHR_ret_1d", "Core_PCE_zscore_60d", "ORCL_zscore_60d", "ORCL_vol_20d", "US6M_Rate_ret_20d", "PCAR_PaccarInc_ret_5d"], "is_new": true}, {"model_id": "new_h7_STRESS_LogisticRegression_N8_t3", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 7, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "LUV_SouthwestAir_ret_5d", "ENB_EnbridgeInc_ret_1d", "DHR_ret_1d", "AORD_AUS_zscore_60d", "IYM_BasicMaterials_ret_20d", "US3Y_Rate_ret_5d"], "is_new": true}, {"model_id": "new_h7_STRESS_LogisticRegression_N8_t4", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 7, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "INTC_ret_1d", "HangSeng_HK_ret_5d", "EMR_Emerson_ret_20d", "QQQ_vol_20d", "EWL_Switzerland_zscore_60d", "3M_ret_5d"], "is_new": true}, {"model_id": "new_h7_STRESS_LogisticRegression_N8_t5", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 7, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "gjr_condvar_h1", "AMD_ret_1d", "SLB_Schlumberger_ret_1d", "Brent_Oil_FRED_ret_5d", "heston_var_ev_h3", "T10Y2Y_Spread_ret_5d"], "is_new": true}, {"model_id": "new_h7_STRESS_LogisticRegression_N8_t6", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 7, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "US3Y_Rate_ret_5d", "IWM_SmallCap_vol_20d", "PFE_ret_1d", "EQIX_Equinix_ret_5d", "Nikkei_Japan_zscore_60d", "PAYX_Paychex_vol_20d"], "is_new": true}, {"model_id": "new_h7_STRESS_LogisticRegression_N8_t7", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 7, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "ASX_Australia_vol_20d", "JNJ_ret_1d", "XLB_Materials_zscore_60d", "CMCSA_ret_1d", "DE_Deere_vol_20d", "heston_var_ev_h7"], "is_new": true}, {"model_id": "new_h7_STRESS_LogisticRegression_N10_t0", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 7, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWY_Korea_zscore_60d", "Nikkei_Japan_vol_20d", "EWG_Germany_ret_20d", "LLY_zscore_60d", "US1Y_Rate_ret_5d", "vix_acceleration_1d", "XLF_Fin_vol_20d", "EWL_Switzerland_vol_20d"], "is_new": true}, {"model_id": "new_h7_STRESS_LogisticRegression_N10_t1", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 7, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "LMT_LockheedMartin_ret_1d", "MS_MorganStanley_ret_1d", "ITT_ITTInc_ret_5d", "SBUX_ret_5d", "TXN_vol_20d", "AMD_ret_1d", "Core_CPI_zscore_60d", "SJM_JM_Smucker_ret_1d"], "is_new": true}, {"model_id": "new_h7_STRESS_LogisticRegression_N10_t2", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 7, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "SLB_Schlumberger_ret_1d", "TXN_vol_20d", "XLV_Health_zscore_60d", "T10Y2Y_Spread_ret_5d", "MO_AltriaMG_ret_1d", "CI_Cigna_vol_20d", "LMT_LockheedMartin_vol_20d", "LLY_zscore_60d"], "is_new": true}, {"model_id": "new_h7_STRESS_LogisticRegression_N10_t3", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 7, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "ORCL_zscore_60d", "spx_vol_5d", "US1Y_Rate_ret_5d", "Brent_Oil_FRED_ret_5d", "NEE_NextEra_ret_20d", "heston_var_ev_h7", "XLK_Tech_zscore_60d", "CTAS_Cintas_vol_20d"], "is_new": true}, {"model_id": "new_h7_STRESS_LogisticRegression_N10_t4", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 7, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWY_Korea_zscore_60d", "AMZN_ret_5d", "FedFunds_zscore_60d", "DAX_Germany_vol_20d", "IYM_BasicMaterials_ret_20d", "EWY_Korea_ret_20d", "LMT_LockheedMartin_ret_1d", "HangSeng_HK_ret_5d"], "is_new": true}, {"model_id": "new_h7_STRESS_LogisticRegression_N10_t5", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 7, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "AXP_Amex_ret_20d", "EQR_Equity_ret_1d", "Nikkei_Japan_vol_20d", "SO_SouthernCo_ret_5d", "US6M_Rate_ret_20d", "spx_abs_ret_max_5d", "EQIX_Equinix_ret_5d", "BDX_Becton_Dickinson_ret_20d"], "is_new": true}, {"model_id": "new_h7_STRESS_LogisticRegression_N10_t6", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 7, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "XLK_Tech_zscore_60d", "EWQ_France_zscore_60d", "LLY_zscore_60d", "PAYX_Paychex_zscore_60d", "HD_zscore_60d", "HD_ret_20d", "BTI_BritishAmerican_ret_20d", "TED_Spread_zscore_60d"], "is_new": true}, {"model_id": "new_h7_STRESS_LogisticRegression_N10_t7", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 7, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "heston_var_ev_h3", "AXP_Amex_ret_20d", "QQQ_vol_20d", "BTI_BritishAmerican_ret_20d", "HangSeng_HK_ret_1d", "Brent_Oil_FRED_ret_20d", "MRK_Merck_zscore_60d", "SJM_JM_Smucker_ret_5d"], "is_new": true}, {"model_id": "new_h7_STRESS_LogisticRegression_N12_t0", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 7, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "PCAR_PaccarInc_ret_5d", "DE_Deere_ret_5d", "Brent_Oil_FRED_ret_5d", "CPB_CampbellSoup_ret_20d", "US1Y_Rate_ret_20d", "PAYX_Paychex_vol_20d", "DIS_vol_20d", "SJM_JM_Smucker_ret_5d", "TGT_Target_zscore_60d", "EXC_Exelon_zscore_60d"], "is_new": true}, {"model_id": "new_h7_STRESS_LogisticRegression_N12_t1", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 7, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWA_Australia_zscore_60d", "MRK_Merck_zscore_60d", "MSTR_Bitcoin3_ret_20d", "LLY_zscore_60d", "CTAS_Cintas_vol_20d", "BTI_BritishAmerican_ret_5d", "NVDA_vol_20d", "WTI_Oil_FRED_zscore_60d", "XLB_Materials_zscore_60d", "SJM_JM_Smucker_ret_5d"], "is_new": true}, {"model_id": "new_h7_STRESS_LogisticRegression_N12_t2", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 7, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "heston_var_ev_h7", "T10Y2Y_Spread_ret_5d", "HangSeng_HK_ret_1d", "AORD_AUS_zscore_60d", "EWM_Malaysia_vol_20d", "CTAS_Cintas_vol_20d", "MRK_Merck_zscore_60d", "XLB_Materials_zscore_60d", "IYR_US_REIT2_zscore_60d", "EWS_Singapore_ret_5d"], "is_new": true}, {"model_id": "new_h7_STRESS_LogisticRegression_N12_t3", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 7, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "T10Y2Y_Spread_ret_5d", "AMZN_ret_5d", "FedFunds_zscore_60d", "vix_mean_abs_ret_5d", "DHR_ret_1d", "MSTR_Bitcoin3_ret_1d", "PAYX_Paychex_zscore_60d", "DHR_vol_20d", "Nikkei_Japan_zscore_60d", "VOD_Vodafone_zscore_60d"], "is_new": true}, {"model_id": "new_h7_STRESS_LogisticRegression_N12_t4", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 7, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "VRP_ma5", "GILD_Gilead_ret_20d", "PCAR_PaccarInc_ret_5d", "GD_GeneralDynamics_zscore_60d", "MSTR_Bitcoin3_ret_1d", "XOM_ret_1d", "ES_Evergy_ret_1d", "IWM_SmallCap_vol_20d", "XLV_Health_zscore_60d", "ORCL_zscore_60d"], "is_new": true}, {"model_id": "new_h7_STRESS_LogisticRegression_N12_t5", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 7, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "hmm_p_stress", "IWM_SmallCap_vol_20d", "TED_Spread_zscore_60d", "GILD_Gilead_ret_20d", "BLK_BlackRock_zscore_60d", "EWH_HongKong_ret_5d", "IYM_BasicMaterials_ret_20d", "DHR_vol_20d", "NEE_NextEra_ret_20d", "DHR_ret_1d"], "is_new": true}, {"model_id": "new_h7_STRESS_LogisticRegression_N12_t6", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 7, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "AMD_ret_5d", "QQQ_vol_20d", "T_ret_1d", "HangSeng_HK_vol_20d", "GD_GeneralDynamics_zscore_60d", "EWQ_France_zscore_60d", "PPL_PPL_ret_1d", "CTAS_Cintas_vol_20d", "AXP_Amex_vol_20d", "AORD_AUS_zscore_60d"], "is_new": true}, {"model_id": "new_h7_STRESS_LogisticRegression_N12_t7", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 7, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "XLK_Tech_zscore_60d", "CMCSA_ret_1d", "DOW_Price_zscore_60d", "BDX_Becton_Dickinson_ret_20d", "EXC_Exelon_zscore_60d", "ASX_Australia_ret_5d", "T_ret_1d", "AXP_Amex_ret_20d", "Michigan_Sentiment_ret_20d", "EWG_Germany_ret_20d"], "is_new": true}, {"model_id": "new_h7_STRESS_LogisticRegression_N15_t0", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 7, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "Industrial_Production_zscore_60d", "SLB_Schlumberger_ret_1d", "NOC_Northrop_ret_20d", "VVIX_ret_20d", "AMT_AmericanTower_ret_1d", "HangSeng_HK_vol_20d", "LOW_Lowes_ret_5d", "CI_Cigna_vol_20d", "EWQ_France_zscore_60d", "EMR_Emerson_ret_20d", "XOM_ret_1d", "EWJ_Japan_vol_20d", "Core_PCE_zscore_60d"], "is_new": true}, {"model_id": "new_h7_STRESS_LogisticRegression_N15_t1", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 7, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "US1Y_Rate_ret_5d", "EWS_Singapore_ret_5d", "HD_ret_20d", "US1Y_Rate_ret_20d", "spx_vol_5d", "EWQ_France_ret_20d", "ASX_Australia_vol_20d", "SLB_Schlumberger_ret_5d", "MSTR_Bitcoin3_ret_20d", "EWQ_France_zscore_60d", "Retail_Sales_zscore_60d", "BLK_BlackRock_zscore_60d", "CPB_CampbellSoup_zscore_60d"], "is_new": true}, {"model_id": "new_h7_STRESS_LogisticRegression_N15_t2", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 7, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "AMGN_Amgen_ret_1d", "ORCL_zscore_60d", "XLB_Materials_zscore_60d", "GD_GeneralDynamics_zscore_60d", "BTI_BritishAmerican_ret_20d", "GILD_Gilead_ret_20d", "DAX_Germany_zscore_60d", "T10Y2Y_Spread_ret_5d", "EQIX_Equinix_ret_5d", "EXC_Exelon_zscore_60d", "TGT_Target_zscore_60d", "EWQ_France_zscore_60d", "CLX_Clorox_vol_20d"], "is_new": true}, {"model_id": "new_h7_STRESS_LogisticRegression_N15_t3", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 7, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "BLK_BlackRock_zscore_60d", "GD_GeneralDynamics_zscore_60d", "AMGN_Amgen_ret_1d", "CCI_CrownCastle_vol_20d", "TXN_vol_20d", "AMD_ret_5d", "spx_momentum_3d", "SBUX_zscore_60d", "XLK_Tech_zscore_60d", "HD_ret_1d", "VVIX_ret_20d", "SLB_Schlumberger_ret_5d", "AMD_ret_1d"], "is_new": true}, {"model_id": "new_h7_STRESS_LogisticRegression_N15_t4", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 7, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "XLY_Disc_vol_20d", "MRK_Merck_zscore_60d", "EWM_Malaysia_ret_1d", "EWY_Korea_zscore_60d", "PPL_PPL_ret_1d", "US3M_Rate_zscore_60d", "LMT_LockheedMartin_ret_1d", "SBUX_ret_5d", "MSTR_Bitcoin3_ret_5d", "IWM_SmallCap_vol_20d", "Brent_Oil_FRED_ret_20d", "EWL_Switzerland_vol_20d", "XLF_Fin_vol_20d"], "is_new": true}, {"model_id": "new_h7_STRESS_LogisticRegression_N15_t5", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 7, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "CPB_CampbellSoup_ret_20d", "EWM_Malaysia_ret_1d", "DAX_Germany_zscore_60d", "CMCSA_ret_1d", "AORD_AUS_zscore_60d", "SBUX_vol_20d", "gjr_condvar_h1", "AMD_ret_5d", "EOG_EOGResources_ret_5d", "HD_ret_20d", "PFE_ret_1d", "DE_Deere_vol_20d", "PCAR_PaccarInc_ret_5d"], "is_new": true}, {"model_id": "new_h7_STRESS_LogisticRegression_N15_t6", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 7, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "HangSeng_HK_ret_1d", "M_Macys_vol_20d", "EWM_Malaysia_vol_20d", "VRP_ma5", "IBEX_Spain_ret_20d", "TED_Spread_zscore_60d", "FedFunds_zscore_60d", "CI_Cigna_vol_20d", "US30Y_Rate_ret_20d", "HUM_Humana_ret_5d", "US3M_Rate_vol_20d", "SLB_Schlumberger_ret_1d", "ORCL_vol_20d"], "is_new": true}, {"model_id": "new_h7_STRESS_LogisticRegression_N15_t7", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 7, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "DIS_vol_20d", "TGT_Target_zscore_60d", "LUV_SouthwestAir_ret_5d", "3M_vol_20d", "CPB_CampbellSoup_vol_20d", "EXC_Exelon_zscore_60d", "ORCL_vol_20d", "TED_Spread_vol_20d", "XLK_Tech_zscore_60d", "PCAR_PaccarInc_ret_5d", "ORCL_zscore_60d", "EWL_Switzerland_vol_20d", "LMT_LockheedMartin_ret_1d"], "is_new": true}, {"model_id": "new_h7_STRESS_LogisticRegression_N20_t0", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 7, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "Nikkei_Japan_zscore_60d", "WTI_Oil_FRED_zscore_60d", "NOC_Northrop_ret_20d", "BTI_BritishAmerican_ret_20d", "PAYX_Paychex_ret_20d", "XOM_ret_20d", "AVB_AvalonBay_zscore_60d", "EQR_Equity_ret_1d", "QQQ_vol_20d", "PLD_Prologis_ret_5d", "LMT_LockheedMartin_ret_1d", "US3Y_Rate_ret_5d", "Brent_Oil_FRED_ret_5d", "CLX_Clorox_vol_20d", "AMT_AmericanTower_ret_1d", "PAYX_Paychex_zscore_60d", "ENB_EnbridgeInc_ret_1d", "EXC_Exelon_ret_1d"], "is_new": true}, {"model_id": "new_h7_STRESS_LogisticRegression_N20_t1", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 7, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "NOC_Northrop_ret_20d", "Retail_Sales_zscore_60d", "3M_vol_20d", "CPB_CampbellSoup_vol_20d", "AMZN_ret_5d", "EWQ_France_ret_20d", "SCHW_Schwab_ret_5d", "PLD_Prologis_ret_5d", "IYR_US_REIT2_zscore_60d", "NVDA_vol_20d", "MO_AltriaMG_ret_1d", "TED_Spread_zscore_60d", "Nikkei_Japan_zscore_60d", "vix_acceleration_1d", "XOM_ret_1d", "INTC_ret_5d", "VOD_Vodafone_zscore_60d", "GILD_Gilead_ret_20d"], "is_new": true}, {"model_id": "new_h7_STRESS_LogisticRegression_N20_t2", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 7, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "ES_Evergy_ret_1d", "EOG_EOGResources_vol_20d", "CPB_CampbellSoup_zscore_60d", "US6M_Rate_ret_20d", "PAYX_Paychex_zscore_60d", "NWL_Newell_ret_20d", "M_Macys_vol_20d", "GILD_Gilead_ret_20d", "LOW_Lowes_ret_5d", "US1Y_Rate_ret_5d", "BA_ret_1d", "PLD_Prologis_ret_5d", "CLX_Clorox_vol_20d", "WTI_Oil_FRED_zscore_60d", "BLK_BlackRock_zscore_60d", "NEE_NextEra_ret_20d", "HD_ret_1d", "IYR_US_REIT2_zscore_60d"], "is_new": true}, {"model_id": "new_h7_STRESS_LogisticRegression_N20_t3", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 7, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "HD_ret_1d", "MS_MorganStanley_zscore_60d", "EFFR_ret_1d", "US7Y_Rate_ret_20d", "LUV_SouthwestAir_ret_5d", "spx_vol_5d", "MSTR_Bitcoin3_ret_20d", "SLB_Schlumberger_ret_1d", "AMZN_ret_5d", "LOW_Lowes_ret_5d", "Brent_Oil_FRED_ret_20d", "BTI_BritishAmerican_ret_5d", "gjr_condvar_h1", "US6M_Rate_ret_20d", "HangSeng_HK_ret_5d", "PCAR_PaccarInc_ret_5d", "DOW_Price_zscore_60d", "INTC_ret_5d"], "is_new": true}, {"model_id": "new_h7_STRESS_LogisticRegression_N20_t4", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 7, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "MSTR_Bitcoin3_ret_5d", "Core_PCE_zscore_60d", "ENB_EnbridgeInc_ret_1d", "Nikkei_Japan_zscore_60d", "vix_mean_abs_ret_5d", "SO_SouthernCo_ret_5d", "JNJ_ret_1d", "EWM_Malaysia_ret_1d", "Brent_Oil_FRED_ret_20d", "HD_ret_20d", "3M_ret_5d", "ASX_Australia_ret_5d", "EQR_Equity_ret_1d", "EWM_Malaysia_zscore_60d", "PPL_PPL_ret_1d", "Nikkei_Japan_vol_20d", "Retail_Sales_zscore_60d", "EWJ_Japan_vol_20d"], "is_new": true}, {"model_id": "new_h7_STRESS_LogisticRegression_N20_t5", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 7, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "PAYX_Paychex_ret_20d", "BTI_BritishAmerican_ret_5d", "EWM_Malaysia_zscore_60d", "MS_MorganStanley_ret_1d", "HD_ret_20d", "GILD_Gilead_ret_20d", "Industrial_Production_zscore_60d", "XLF_Fin_vol_20d", "EQIX_Equinix_ret_5d", "EWS_Singapore_ret_5d", "NWL_Newell_ret_20d", "EXC_Exelon_ret_1d", "US3M_Rate_vol_20d", "XLB_Materials_zscore_60d", "SBUX_vol_20d", "VOD_Vodafone_zscore_60d", "TM_Telephone_vol_20d", "LOW_Lowes_ret_5d"], "is_new": true}, {"model_id": "new_h7_STRESS_LogisticRegression_N20_t6", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 7, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "BTI_BritishAmerican_ret_5d", "ENB_EnbridgeInc_ret_1d", "JNJ_ret_1d", "LLY_zscore_60d", "XLB_Materials_zscore_60d", "INTC_ret_5d", "NOC_Northrop_ret_20d", "VVIX_ret_20d", "ES_Evergy_ret_1d", "EWY_Korea_ret_20d", "T10Y2Y_Spread_ret_5d", "EWM_Malaysia_vol_20d", "CTAS_Cintas_vol_20d", "GD_GeneralDynamics_zscore_60d", "EOG_EOGResources_vol_20d", "MRK_Merck_zscore_60d", "MSTR_Bitcoin3_ret_5d", "ASX_Australia_vol_20d"], "is_new": true}, {"model_id": "new_h7_STRESS_LogisticRegression_N20_t7", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 7, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "PPL_PPL_ret_1d", "EWG_Germany_vol_20d", "EWS_Singapore_ret_5d", "XLY_Disc_vol_20d", "CPB_CampbellSoup_vol_20d", "SJM_JM_Smucker_ret_5d", "EXC_Exelon_ret_1d", "hmm_p_stress", "TED_Spread_vol_20d", "NOC_Northrop_ret_20d", "AMT_AmericanTower_ret_1d", "Nikkei_Japan_zscore_60d", "BTI_BritishAmerican_ret_5d", "IYM_BasicMaterials_ret_20d", "heston_var_ev_h5", "AMGN_Amgen_ret_1d", "GILD_Gilead_ret_20d", "LOW_Lowes_ret_5d"], "is_new": true}, {"model_id": "new_h7_STRESS_LogisticRegression_N25_t0", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 7, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "VOD_Vodafone_zscore_60d", "AVB_AvalonBay_zscore_60d", "AORD_AUS_zscore_60d", "EMR_Emerson_ret_20d", "BLK_BlackRock_zscore_60d", "SLB_Schlumberger_ret_1d", "XLB_Materials_zscore_60d", "CPB_CampbellSoup_zscore_60d", "EXC_Exelon_zscore_60d", "BDX_Becton_Dickinson_ret_20d", "VRP_ma5", "NOC_Northrop_ret_20d", "HD_ret_1d", "Michigan_Sentiment_ret_20d", "MSTR_Bitcoin3_ret_1d", "GILD_Gilead_ret_20d", "EQR_Equity_ret_1d", "HangSeng_HK_vol_20d", "US3M_Rate_vol_20d", "HD_ret_20d", "Brent_Oil_FRED_ret_20d", "TM_Telephone_vol_20d", "EOG_EOGResources_ret_5d"], "is_new": true}, {"model_id": "new_h7_STRESS_LogisticRegression_N25_t1", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 7, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "AVB_AvalonBay_zscore_60d", "PLD_Prologis_ret_5d", "CTAS_Cintas_vol_20d", "HangSeng_HK_ret_5d", "gjr_condvar_h1", "CI_Cigna_vol_20d", "PAYX_Paychex_vol_20d", "HangSeng_HK_vol_20d", "SJM_JM_Smucker_ret_1d", "heston_ev_h3", "CPB_CampbellSoup_zscore_60d", "LUV_SouthwestAir_ret_5d", "PG_ret_20d", "GILD_Gilead_ret_20d", "VOD_Vodafone_zscore_60d", "VRP_ma5", "M_Macys_vol_20d", "FedFunds_zscore_60d", "PFE_ret_1d", "ENB_EnbridgeInc_ret_1d", "EWL_Switzerland_vol_20d", "ORCL_vol_20d", "MS_MorganStanley_ret_5d"], "is_new": true}, {"model_id": "new_h7_STRESS_LogisticRegression_N25_t2", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 7, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "Brent_Oil_FRED_ret_20d", "GE_ret_1d", "SCHW_Schwab_ret_5d", "AMD_ret_5d", "ENB_EnbridgeInc_ret_1d", "PAYX_Paychex_vol_20d", "XLK_Tech_zscore_60d", "EXC_Exelon_zscore_60d", "BLK_BlackRock_zscore_60d", "EXC_Exelon_ret_1d", "VOD_Vodafone_zscore_60d", "TED_Spread_vol_20d", "MS_MorganStanley_ret_1d", "vix_mean_abs_ret_5d", "CPB_CampbellSoup_ret_5d", "CPB_CampbellSoup_vol_20d", "hmm_p_stress", "EOG_EOGResources_ret_5d", "GD_GeneralDynamics_zscore_60d", "PAYX_Paychex_zscore_60d", "US1Y_Rate_ret_20d", "DAX_Germany_vol_20d", "heston_ev_h3"], "is_new": true}, {"model_id": "new_h7_STRESS_LogisticRegression_N25_t3", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 7, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "GILD_Gilead_ret_20d", "MSTR_Bitcoin3_ret_20d", "CCI_CrownCastle_vol_20d", "XOM_ret_1d", "heston_var_ev_h5", "EOG_EOGResources_vol_20d", "EWY_Korea_zscore_60d", "ASX_Australia_ret_5d", "3M_ret_5d", "CMCSA_ret_1d", "DHR_ret_1d", "AXP_Amex_vol_20d", "EFFR_ret_1d", "US6M_Rate_ret_20d", "SJM_JM_Smucker_ret_5d", "US7Y_Rate_ret_20d", "LMT_LockheedMartin_ret_1d", "LOW_Lowes_ret_20d", "HangSeng_HK_vol_20d", "TED_Spread_vol_20d", "DAX_Germany_vol_20d", "PFE_ret_1d", "hmm_p_stress"], "is_new": true}, {"model_id": "new_h7_STRESS_LogisticRegression_N25_t4", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 7, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "vix_acceleration_1d", "DHR_vol_20d", "US3Y_Rate_ret_5d", "EWG_Germany_vol_20d", "HD_ret_20d", "AVB_AvalonBay_zscore_60d", "EWY_Korea_zscore_60d", "PCAR_PaccarInc_ret_5d", "ENB_EnbridgeInc_ret_1d", "CMCSA_ret_1d", "Brent_Oil_FRED_ret_20d", "PPL_PPL_ret_1d", "heston_var_ev_h7", "MO_AltriaMG_ret_1d", "NWL_Newell_ret_20d", "BLK_BlackRock_zscore_60d", "DIS_vol_20d", "NOC_Northrop_ret_20d", "XOM_ret_20d", "BTI_BritishAmerican_ret_5d", "IBEX_Spain_ret_20d", "ASX_Australia_ret_5d", "M_Macys_vol_20d"], "is_new": true}, {"model_id": "new_h7_STRESS_LogisticRegression_N25_t5", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 7, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "BTI_BritishAmerican_ret_20d", "PAYX_Paychex_ret_20d", "US1Y_Rate_ret_20d", "MSTR_Bitcoin3_ret_20d", "EWY_Korea_zscore_60d", "spx_vol_5d", "WTI_Oil_FRED_zscore_60d", "PAYX_Paychex_zscore_60d", "CPB_CampbellSoup_ret_20d", "spx_abs_ret_max_5d", "EWY_Korea_ret_20d", "heston_var_ev_h7", "ORCL_vol_20d", "NFCI_ret_5d", "ASX_Australia_vol_20d", "3M_ret_5d", "US5Y_Rate_ret_5d", "SO_SouthernCo_ret_5d", "MS_MorganStanley_ret_1d", "heston_var_ev_h5", "TM_Telephone_ret_1d", "XLK_Tech_zscore_60d", "AMT_AmericanTower_ret_1d"], "is_new": true}, {"model_id": "new_h7_STRESS_LogisticRegression_N25_t6", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 7, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "AVB_AvalonBay_zscore_60d", "EWA_Australia_ret_1d", "HD_zscore_60d", "XLV_Health_zscore_60d", "AMD_ret_5d", "EQIX_Equinix_ret_5d", "LOW_Lowes_ret_20d", "SBUX_ret_5d", "Industrial_Production_zscore_60d", "CLX_Clorox_vol_20d", "BLK_BlackRock_zscore_60d", "US30Y_Rate_ret_20d", "CCI_CrownCastle_vol_20d", "US7Y_Rate_ret_20d", "EWJ_Japan_vol_20d", "XLF_Fin_vol_20d", "BA_ret_1d", "DAX_Germany_vol_20d", "LLY_zscore_60d", "HangSeng_HK_vol_20d", "AMT_AmericanTower_ret_1d", "INTC_ret_5d", "MS_MorganStanley_zscore_60d"], "is_new": true}, {"model_id": "new_h7_STRESS_LogisticRegression_N25_t7", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 7, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "CPB_CampbellSoup_ret_5d", "US3M_Rate_zscore_60d", "SBUX_zscore_60d", "IWM_SmallCap_vol_20d", "EWG_Germany_vol_20d", "XLB_Materials_zscore_60d", "TM_Telephone_vol_20d", "vix_acceleration_1d", "LMT_LockheedMartin_ret_1d", "HUM_Humana_ret_5d", "PAYX_Paychex_zscore_60d", "NVDA_vol_20d", "WTI_Oil_FRED_zscore_60d", "AMD_ret_1d", "US5Y_Rate_ret_5d", "GD_GeneralDynamics_zscore_60d", "CMCSA_ret_1d", "Industrial_Production_zscore_60d", "SPY_zscore_60d", "MSTR_Bitcoin3_ret_5d", "EWM_Malaysia_ret_1d", "CPB_CampbellSoup_vol_20d", "XLV_Health_zscore_60d"], "is_new": true}, {"model_id": "new_h7_STRESS_LogisticRegression_N30_t0", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 7, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "SJM_JM_Smucker_ret_1d", "US7Y_Rate_ret_20d", "BLK_BlackRock_zscore_60d", "NEE_NextEra_ret_20d", "EQR_Equity_ret_1d", "EFFR_vol_20d", "GD_GeneralDynamics_zscore_60d", "T_ret_1d", "Brent_Oil_FRED_ret_5d", "EWQ_France_ret_20d", "US5Y_Rate_ret_5d", "DE_Deere_vol_20d", "VOD_Vodafone_zscore_60d", "EXC_Exelon_zscore_60d", "ES_Evergy_ret_1d", "SPY_zscore_60d", "SBUX_ret_5d", "DAX_Germany_zscore_60d", "SBUX_zscore_60d", "NOC_Northrop_ret_20d", "ITT_ITTInc_ret_5d", "BTI_BritishAmerican_ret_5d", "Core_PCE_zscore_60d", "HangSeng_HK_ret_1d", "ASX_Australia_ret_5d", "EMR_Emerson_ret_20d", "AVB_AvalonBay_zscore_60d", "AMGN_Amgen_ret_1d"], "is_new": true}, {"model_id": "new_h7_STRESS_LogisticRegression_N30_t1", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 7, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "gjr_condvar_h1", "AMZN_ret_5d", "MSTR_Bitcoin3_ret_20d", "LUV_SouthwestAir_ret_5d", "ENB_EnbridgeInc_ret_1d", "MSTR_Bitcoin3_ret_1d", "SBUX_ret_5d", "SBUX_zscore_60d", "Michigan_Sentiment_ret_20d", "PFE_ret_1d", "Nikkei_Japan_vol_20d", "EWQ_France_zscore_60d", "US5Y_Rate_ret_5d", "Core_CPI_zscore_60d", "PG_ret_20d", "Brent_Oil_FRED_ret_20d", "AORD_AUS_zscore_60d", "HD_zscore_60d", "EWY_Korea_ret_20d", "EWY_Korea_zscore_60d", "BTI_BritishAmerican_ret_20d", "ASX_Australia_ret_5d", "MS_MorganStanley_zscore_60d", "DE_Deere_ret_5d", "SLB_Schlumberger_ret_1d", "M_Macys_vol_20d", "VVIX_ret_20d", "SJM_JM_Smucker_ret_1d"], "is_new": true}, {"model_id": "new_h7_STRESS_LogisticRegression_N30_t2", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 7, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "Retail_Sales_zscore_60d", "MSTR_Bitcoin3_ret_5d", "CCI_CrownCastle_vol_20d", "CPB_CampbellSoup_ret_5d", "FedFunds_zscore_60d", "ORCL_zscore_60d", "heston_ev_h3", "SPY_zscore_60d", "SO_SouthernCo_ret_5d", "DE_Deere_ret_5d", "HangSeng_HK_vol_20d", "EXC_Exelon_ret_1d", "TM_Telephone_vol_20d", "spx_momentum_3d", "IWM_SmallCap_vol_20d", "ORCL_vol_20d", "HD_ret_1d", "heston_var_ev_h7", "M_Macys_vol_20d", "HUM_Humana_ret_5d", "EWC_Canada_zscore_60d", "Nikkei_Japan_vol_20d", "GE_ret_1d", "LMT_LockheedMartin_vol_20d", "CPB_CampbellSoup_zscore_60d", "LMT_LockheedMartin_ret_1d", "Core_CPI_zscore_60d", "Brent_Oil_FRED_ret_20d"], "is_new": true}, {"model_id": "new_h7_STRESS_LogisticRegression_N30_t3", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 7, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "HD_zscore_60d", "SO_SouthernCo_ret_5d", "XLY_Disc_vol_20d", "ORCL_zscore_60d", "Brent_Oil_FRED_ret_20d", "US3Y_Rate_ret_5d", "IYM_BasicMaterials_ret_20d", "IYR_US_REIT2_zscore_60d", "CTAS_Cintas_vol_20d", "FedFunds_zscore_60d", "MSTR_Bitcoin3_ret_20d", "EQR_Equity_ret_1d", "EWY_Korea_ret_20d", "AMGN_Amgen_ret_1d", "Core_CPI_zscore_60d", "EWL_Switzerland_vol_20d", "SCHW_Schwab_ret_5d", "XLV_Health_zscore_60d", "US1Y_Rate_ret_20d", "VVIX_ret_20d", "EWA_Australia_zscore_60d", "DIS_vol_20d", "PLD_Prologis_ret_5d", "MS_MorganStanley_zscore_60d", "NWL_Newell_ret_20d", "MO_AltriaMG_ret_1d", "DE_Deere_vol_20d", "NFCI_ret_5d"], "is_new": true}, {"model_id": "new_h7_STRESS_LogisticRegression_N30_t4", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 7, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "PAYX_Paychex_ret_20d", "HUM_Humana_ret_5d", "HangSeng_HK_vol_20d", "AXP_Amex_ret_20d", "CPB_CampbellSoup_ret_20d", "GILD_Gilead_ret_20d", "EWA_Australia_ret_1d", "M_Macys_vol_20d", "NWL_Newell_ret_20d", "EWY_Korea_zscore_60d", "EXC_Exelon_ret_1d", "Nikkei_Japan_vol_20d", "US3M_Rate_zscore_60d", "AMD_ret_5d", "EMR_Emerson_ret_20d", "hmm_p_stress", "XOM_ret_20d", "heston_var_ev_h5", "NVDA_vol_20d", "HD_ret_1d", "LMT_LockheedMartin_vol_20d", "spx_momentum_3d", "DAX_Germany_zscore_60d", "US5Y_Rate_ret_5d", "US3M_Rate_vol_20d", "PFE_ret_1d", "Brent_Oil_FRED_ret_5d", "SLB_Schlumberger_ret_1d"], "is_new": true}, {"model_id": "new_h7_STRESS_LogisticRegression_N30_t5", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 7, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "WTI_Oil_FRED_zscore_60d", "EXC_Exelon_zscore_60d", "ORCL_zscore_60d", "spx_abs_ret_max_5d", "DAX_Germany_zscore_60d", "EWQ_France_zscore_60d", "IYR_US_REIT2_zscore_60d", "ENB_EnbridgeInc_ret_1d", "EXC_Exelon_ret_1d", "TED_Spread_vol_20d", "TED_Spread_zscore_60d", "AXP_Amex_ret_20d", "Retail_Sales_zscore_60d", "CPB_CampbellSoup_ret_20d", "XLK_Tech_zscore_60d", "Michigan_Sentiment_ret_20d", "EWM_Malaysia_vol_20d", "CMCSA_ret_1d", "HangSeng_HK_ret_5d", "DHR_vol_20d", "US3M_Rate_zscore_60d", "BTI_BritishAmerican_ret_20d", "gjr_condvar_h1", "heston_var_ev_h7", "Brent_Oil_FRED_ret_5d", "MS_MorganStanley_ret_1d", "SO_SouthernCo_ret_5d", "MSTR_Bitcoin3_ret_5d"], "is_new": true}, {"model_id": "new_h7_STRESS_LogisticRegression_N30_t6", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 7, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWM_Malaysia_vol_20d", "US3M_Rate_zscore_60d", "BA_ret_1d", "heston_ev_h3", "T10Y2Y_Spread_ret_5d", "Core_CPI_zscore_60d", "LMT_LockheedMartin_vol_20d", "XLF_Fin_vol_20d", "PLD_Prologis_ret_5d", "EWA_Australia_ret_1d", "DOW_Price_zscore_60d", "GD_GeneralDynamics_zscore_60d", "US30Y_Rate_ret_20d", "spx_momentum_3d", "CTAS_Cintas_vol_20d", "XLK_Tech_zscore_60d", "CI_Cigna_vol_20d", "ITT_ITTInc_ret_5d", "VRP_ma5", "EXC_Exelon_zscore_60d", "ES_Evergy_ret_1d", "US3Y_Rate_ret_5d", "MS_MorganStanley_ret_5d", "TXN_vol_20d", "HD_ret_20d", "LOW_Lowes_ret_20d", "SCHW_Schwab_ret_5d", "DAX_Germany_zscore_60d"], "is_new": true}, {"model_id": "new_h7_STRESS_LogisticRegression_N30_t7", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 7, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "AMGN_Amgen_ret_1d", "SBUX_zscore_60d", "CPB_CampbellSoup_zscore_60d", "SPY_zscore_60d", "SCHW_Schwab_ret_5d", "AMT_AmericanTower_ret_1d", "VVIX_ret_20d", "PAYX_Paychex_ret_20d", "JNJ_ret_1d", "DE_Deere_vol_20d", "US1Y_Rate_ret_20d", "IWM_SmallCap_vol_20d", "BTI_BritishAmerican_ret_20d", "EWC_Canada_zscore_60d", "DAX_Germany_zscore_60d", "XLK_Tech_zscore_60d", "vix_mean_abs_ret_5d", "US3M_Rate_vol_20d", "PG_ret_20d", "Core_CPI_zscore_60d", "EWQ_France_zscore_60d", "ORCL_zscore_60d", "EWJ_Japan_vol_20d", "gjr_condvar_h1", "TXN_vol_20d", "HD_ret_1d", "INTC_ret_5d", "spx_momentum_3d"], "is_new": true}, {"model_id": "new_h7_GLOBAL_XGBoost_N5_t0", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 7, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "US30Y_Rate_ret_20d", "HUM_Humana_ret_5d", "XLF_Fin_vol_20d"], "is_new": true}, {"model_id": "new_h7_GLOBAL_XGBoost_N5_t1", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 7, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "US5Y_Rate_ret_5d", "CMCSA_ret_1d", "EWJ_Japan_vol_20d"], "is_new": true}, {"model_id": "new_h7_GLOBAL_XGBoost_N5_t2", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 7, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "US30Y_Rate_ret_20d", "EWS_Singapore_ret_5d", "DHR_vol_20d"], "is_new": true}, {"model_id": "new_h7_GLOBAL_XGBoost_N5_t3", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 7, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "MSTR_Bitcoin3_ret_20d", "FedFunds_zscore_60d", "Industrial_Production_zscore_60d"], "is_new": true}, {"model_id": "new_h7_GLOBAL_XGBoost_N5_t4", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 7, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "T_ret_1d", "MRK_Merck_zscore_60d", "PAYX_Paychex_ret_20d"], "is_new": true}, {"model_id": "new_h7_GLOBAL_XGBoost_N5_t5", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 7, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "GD_GeneralDynamics_zscore_60d", "ASX_Australia_ret_5d", "LMT_LockheedMartin_vol_20d"], "is_new": true}, {"model_id": "new_h7_GLOBAL_XGBoost_N5_t6", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 7, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "AXP_Amex_vol_20d", "HangSeng_HK_ret_5d", "heston_var_ev_h3"], "is_new": true}, {"model_id": "new_h7_GLOBAL_XGBoost_N5_t7", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 7, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "PG_ret_20d", "EWJ_Japan_vol_20d", "LOW_Lowes_ret_5d"], "is_new": true}, {"model_id": "new_h7_GLOBAL_XGBoost_N8_t0", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 7, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "HangSeng_HK_vol_20d", "spx_vol_5d", "LOW_Lowes_ret_20d", "Industrial_Production_zscore_60d", "EWQ_France_zscore_60d", "heston_ev_h3"], "is_new": true}, {"model_id": "new_h7_GLOBAL_XGBoost_N8_t1", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 7, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "US5Y_Rate_ret_5d", "US1Y_Rate_ret_5d", "MS_MorganStanley_ret_5d", "NOC_Northrop_ret_20d", "spx_abs_ret_max_5d", "EXC_Exelon_zscore_60d"], "is_new": true}, {"model_id": "new_h7_GLOBAL_XGBoost_N8_t2", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 7, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "PPL_PPL_ret_1d", "AMGN_Amgen_ret_1d", "PAYX_Paychex_ret_20d", "GE_ret_1d", "EWG_Germany_ret_20d", "SJM_JM_Smucker_ret_5d"], "is_new": true}, {"model_id": "new_h7_GLOBAL_XGBoost_N8_t3", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 7, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EQIX_Equinix_ret_5d", "US3M_Rate_vol_20d", "SO_SouthernCo_ret_5d", "BA_ret_1d", "WTI_Oil_FRED_zscore_60d", "NFCI_ret_5d"], "is_new": true}, {"model_id": "new_h7_GLOBAL_XGBoost_N8_t4", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 7, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EQIX_Equinix_ret_5d", "BTI_BritishAmerican_ret_20d", "MSTR_Bitcoin3_ret_5d", "SO_SouthernCo_ret_5d", "IWM_SmallCap_vol_20d", "CI_Cigna_vol_20d"], "is_new": true}, {"model_id": "new_h7_GLOBAL_XGBoost_N8_t5", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 7, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "SJM_JM_Smucker_ret_1d", "EFFR_ret_1d", "3M_vol_20d", "T10Y2Y_Spread_ret_5d", "US6M_Rate_ret_20d", "PG_ret_20d"], "is_new": true}, {"model_id": "new_h7_GLOBAL_XGBoost_N8_t6", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 7, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "CCI_CrownCastle_vol_20d", "Nikkei_Japan_zscore_60d", "HangSeng_HK_ret_5d", "MRK_Merck_zscore_60d", "vix_mean_abs_ret_5d", "MS_MorganStanley_ret_5d"], "is_new": true}, {"model_id": "new_h7_GLOBAL_XGBoost_N8_t7", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 7, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "MS_MorganStanley_zscore_60d", "CI_Cigna_vol_20d", "DAX_Germany_vol_20d", "WTI_Oil_FRED_zscore_60d", "US5Y_Rate_ret_5d", "Industrial_Production_zscore_60d"], "is_new": true}, {"model_id": "new_h7_GLOBAL_XGBoost_N10_t0", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 7, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "vix_acceleration_1d", "ORCL_zscore_60d", "hmm_p_stress", "TGT_Target_zscore_60d", "BTI_BritishAmerican_ret_5d", "ASX_Australia_ret_5d", "TM_Telephone_ret_1d", "EWY_Korea_ret_20d"], "is_new": true}, {"model_id": "new_h7_GLOBAL_XGBoost_N10_t1", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 7, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "LOW_Lowes_ret_5d", "EWQ_France_zscore_60d", "SBUX_ret_5d", "CLX_Clorox_vol_20d", "HD_ret_20d", "ENB_EnbridgeInc_ret_1d", "EMR_Emerson_ret_20d", "PPL_PPL_ret_1d"], "is_new": true}, {"model_id": "new_h7_GLOBAL_XGBoost_N10_t2", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 7, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "ENB_EnbridgeInc_ret_1d", "MS_MorganStanley_ret_5d", "GILD_Gilead_ret_20d", "EWL_Switzerland_zscore_60d", "EWM_Malaysia_vol_20d", "DE_Deere_ret_5d", "PCAR_PaccarInc_ret_5d", "TXN_vol_20d"], "is_new": true}, {"model_id": "new_h7_GLOBAL_XGBoost_N10_t3", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 7, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EXC_Exelon_ret_1d", "VOD_Vodafone_zscore_60d", "DAX_Germany_zscore_60d", "MSTR_Bitcoin3_ret_20d", "DHR_vol_20d", "SBUX_vol_20d", "PFE_ret_1d", "PAYX_Paychex_vol_20d"], "is_new": true}, {"model_id": "new_h7_GLOBAL_XGBoost_N10_t4", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 7, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "Nikkei_Japan_vol_20d", "AXP_Amex_ret_20d", "EWQ_France_zscore_60d", "CPB_CampbellSoup_vol_20d", "NWL_Newell_ret_20d", "LOW_Lowes_ret_5d", "ASX_Australia_vol_20d", "SO_SouthernCo_ret_5d"], "is_new": true}, {"model_id": "new_h7_GLOBAL_XGBoost_N10_t5", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 7, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "SJM_JM_Smucker_ret_1d", "EWL_Switzerland_zscore_60d", "US5Y_Rate_ret_5d", "XLB_Materials_zscore_60d", "DE_Deere_vol_20d", "heston_var_ev_h5", "ASX_Australia_vol_20d", "INTC_ret_1d"], "is_new": true}, {"model_id": "new_h7_GLOBAL_XGBoost_N10_t6", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 7, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "SLB_Schlumberger_ret_5d", "US1Y_Rate_ret_20d", "T_ret_1d", "PCAR_PaccarInc_ret_5d", "EWY_Korea_zscore_60d", "VVIX_ret_20d", "EXC_Exelon_ret_1d", "EXC_Exelon_zscore_60d"], "is_new": true}, {"model_id": "new_h7_GLOBAL_XGBoost_N10_t7", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 7, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "SPY_zscore_60d", "DAX_Germany_vol_20d", "BTI_BritishAmerican_ret_5d", "HD_ret_1d", "BDX_Becton_Dickinson_ret_20d", "hmm_p_stress", "SBUX_ret_5d", "EWS_Singapore_ret_5d"], "is_new": true}, {"model_id": "new_h7_GLOBAL_XGBoost_N12_t0", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 7, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "NOC_Northrop_ret_20d", "TED_Spread_vol_20d", "EWL_Switzerland_zscore_60d", "EOG_EOGResources_vol_20d", "M_Macys_vol_20d", "Retail_Sales_zscore_60d", "AMGN_Amgen_ret_1d", "GD_GeneralDynamics_zscore_60d", "LLY_zscore_60d", "ORCL_vol_20d"], "is_new": true}, {"model_id": "new_h7_GLOBAL_XGBoost_N12_t1", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 7, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "HUM_Humana_ret_5d", "SPY_zscore_60d", "MS_MorganStanley_zscore_60d", "IYR_US_REIT2_zscore_60d", "EWY_Korea_zscore_60d", "PG_ret_20d", "DE_Deere_ret_5d", "EWH_HongKong_ret_5d", "BA_ret_1d", "EWA_Australia_zscore_60d"], "is_new": true}, {"model_id": "new_h7_GLOBAL_XGBoost_N12_t2", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 7, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "ASX_Australia_vol_20d", "LOW_Lowes_ret_20d", "Nikkei_Japan_vol_20d", "SBUX_vol_20d", "Nikkei_Japan_zscore_60d", "SO_SouthernCo_ret_5d", "BDX_Becton_Dickinson_ret_20d", "EFFR_vol_20d", "AMGN_Amgen_ret_1d", "EWG_Germany_ret_20d"], "is_new": true}, {"model_id": "new_h7_GLOBAL_XGBoost_N12_t3", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 7, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWS_Singapore_ret_5d", "CLX_Clorox_vol_20d", "DE_Deere_vol_20d", "EMR_Emerson_ret_20d", "ORCL_zscore_60d", "T_ret_1d", "BDX_Becton_Dickinson_ret_20d", "SO_SouthernCo_ret_5d", "CCI_CrownCastle_vol_20d", "AMD_ret_1d"], "is_new": true}, {"model_id": "new_h7_GLOBAL_XGBoost_N12_t4", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 7, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "HangSeng_HK_vol_20d", "GILD_Gilead_ret_20d", "IBEX_Spain_ret_20d", "T10Y2Y_Spread_ret_5d", "gjr_condvar_h1", "AVB_AvalonBay_zscore_60d", "PG_ret_20d", "QQQ_vol_20d", "EWA_Australia_zscore_60d", "SJM_JM_Smucker_ret_5d"], "is_new": true}, {"model_id": "new_h7_GLOBAL_XGBoost_N12_t5", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 7, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWA_Australia_ret_1d", "MS_MorganStanley_ret_5d", "IWM_SmallCap_vol_20d", "EWL_Switzerland_zscore_60d", "INTC_ret_1d", "VRP_ma5", "Michigan_Sentiment_ret_20d", "US1Y_Rate_ret_20d", "INTC_ret_5d", "MS_MorganStanley_zscore_60d"], "is_new": true}, {"model_id": "new_h7_GLOBAL_XGBoost_N12_t6", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 7, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "AXP_Amex_ret_20d", "INTC_ret_5d", "EFFR_vol_20d", "M_Macys_vol_20d", "SLB_Schlumberger_ret_5d", "NFCI_ret_5d", "SBUX_vol_20d", "MSTR_Bitcoin3_ret_20d", "MO_AltriaMG_ret_1d", "NEE_NextEra_ret_20d"], "is_new": true}, {"model_id": "new_h7_GLOBAL_XGBoost_N12_t7", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 7, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "AMD_ret_1d", "US3M_Rate_zscore_60d", "EWH_HongKong_ret_5d", "AORD_AUS_zscore_60d", "EWS_Singapore_ret_5d", "PLD_Prologis_ret_5d", "IBEX_Spain_ret_20d", "AXP_Amex_ret_20d", "gjr_condvar_h1", "ORCL_vol_20d"], "is_new": true}, {"model_id": "new_h7_GLOBAL_XGBoost_N15_t0", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 7, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "WTI_Oil_FRED_zscore_60d", "GILD_Gilead_ret_20d", "T_ret_1d", "GD_GeneralDynamics_zscore_60d", "BDX_Becton_Dickinson_ret_20d", "EWM_Malaysia_zscore_60d", "CPB_CampbellSoup_ret_5d", "EWM_Malaysia_vol_20d", "heston_var_ev_h5", "EWY_Korea_ret_20d", "T10Y2Y_Spread_ret_5d", "EWA_Australia_ret_1d", "QQQ_vol_20d"], "is_new": true}, {"model_id": "new_h7_GLOBAL_XGBoost_N15_t1", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 7, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "US1Y_Rate_ret_20d", "GD_GeneralDynamics_zscore_60d", "EOG_EOGResources_ret_5d", "TED_Spread_zscore_60d", "IWM_SmallCap_vol_20d", "CLX_Clorox_vol_20d", "ORCL_zscore_60d", "XOM_ret_20d", "SCHW_Schwab_ret_5d", "LOW_Lowes_ret_5d", "Nikkei_Japan_vol_20d", "IYM_BasicMaterials_ret_20d", "MS_MorganStanley_zscore_60d"], "is_new": true}, {"model_id": "new_h7_GLOBAL_XGBoost_N15_t2", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 7, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "TXN_vol_20d", "CI_Cigna_vol_20d", "MS_MorganStanley_ret_1d", "IYM_BasicMaterials_ret_20d", "EXC_Exelon_zscore_60d", "ORCL_vol_20d", "EWQ_France_ret_20d", "MSTR_Bitcoin3_ret_1d", "SO_SouthernCo_ret_5d", "BLK_BlackRock_zscore_60d", "LOW_Lowes_ret_20d", "ENB_EnbridgeInc_ret_1d", "EXC_Exelon_ret_1d"], "is_new": true}, {"model_id": "new_h7_GLOBAL_XGBoost_N15_t3", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 7, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "ORCL_zscore_60d", "AXP_Amex_ret_20d", "MSTR_Bitcoin3_ret_1d", "heston_var_ev_h5", "HD_ret_5d", "US5Y_Rate_ret_5d", "LOW_Lowes_ret_5d", "DOW_Price_zscore_60d", "hmm_p_stress", "SJM_JM_Smucker_ret_1d", "CI_Cigna_vol_20d", "MSTR_Bitcoin3_ret_5d", "LMT_LockheedMartin_ret_1d"], "is_new": true}, {"model_id": "new_h7_GLOBAL_XGBoost_N15_t4", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 7, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "heston_ev_h3", "spx_momentum_3d", "Michigan_Sentiment_ret_20d", "ORCL_vol_20d", "EWY_Korea_zscore_60d", "EOG_EOGResources_vol_20d", "NWL_Newell_ret_20d", "IYM_BasicMaterials_ret_20d", "Brent_Oil_FRED_ret_20d", "US6M_Rate_ret_20d", "PLD_Prologis_ret_5d", "Nikkei_Japan_zscore_60d", "DHR_ret_1d"], "is_new": true}, {"model_id": "new_h7_GLOBAL_XGBoost_N15_t5", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 7, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWQ_France_zscore_60d", "INTC_ret_1d", "3M_ret_5d", "INTC_ret_5d", "Retail_Sales_zscore_60d", "IYM_BasicMaterials_ret_20d", "US5Y_Rate_ret_5d", "CPB_CampbellSoup_vol_20d", "QQQ_vol_20d", "SJM_JM_Smucker_ret_5d", "heston_ev_h3", "SBUX_vol_20d", "NEE_NextEra_ret_20d"], "is_new": true}, {"model_id": "new_h7_GLOBAL_XGBoost_N15_t6", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 7, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "T_ret_1d", "EWM_Malaysia_ret_1d", "EWG_Germany_ret_20d", "DHR_ret_1d", "EWH_HongKong_ret_5d", "spx_vol_5d", "SLB_Schlumberger_ret_5d", "EWQ_France_zscore_60d", "LOW_Lowes_ret_20d", "ORCL_vol_20d", "EMR_Emerson_ret_20d", "SBUX_vol_20d", "EWJ_Japan_vol_20d"], "is_new": true}, {"model_id": "new_h7_GLOBAL_XGBoost_N15_t7", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 7, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "SLB_Schlumberger_ret_5d", "EFFR_ret_1d", "3M_vol_20d", "M_Macys_vol_20d", "3M_ret_5d", "HUM_Humana_ret_5d", "LMT_LockheedMartin_vol_20d", "HangSeng_HK_ret_1d", "AORD_AUS_zscore_60d", "XLB_Materials_zscore_60d", "Nikkei_Japan_zscore_60d", "XOM_ret_20d", "EQIX_Equinix_ret_5d"], "is_new": true}, {"model_id": "new_h7_GLOBAL_XGBoost_N20_t0", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 7, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWH_HongKong_ret_5d", "HD_ret_20d", "TXN_vol_20d", "EOG_EOGResources_ret_5d", "NVDA_vol_20d", "MO_AltriaMG_ret_1d", "IYR_US_REIT2_zscore_60d", "CI_Cigna_vol_20d", "DE_Deere_vol_20d", "TM_Telephone_ret_1d", "heston_var_ev_h5", "ORCL_zscore_60d", "US1Y_Rate_ret_20d", "EWA_Australia_zscore_60d", "HD_ret_1d", "SLB_Schlumberger_ret_5d", "Brent_Oil_FRED_ret_20d", "US5Y_Rate_ret_5d"], "is_new": true}, {"model_id": "new_h7_GLOBAL_XGBoost_N20_t1", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 7, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "M_Macys_vol_20d", "EWM_Malaysia_zscore_60d", "XLF_Fin_vol_20d", "QQQ_vol_20d", "AORD_AUS_zscore_60d", "AMD_ret_5d", "EWC_Canada_zscore_60d", "IYM_BasicMaterials_ret_20d", "LMT_LockheedMartin_ret_1d", "ENB_EnbridgeInc_ret_1d", "XOM_ret_20d", "US3M_Rate_zscore_60d", "PCAR_PaccarInc_ret_5d", "BA_ret_1d", "VVIX_ret_20d", "Michigan_Sentiment_ret_20d", "CI_Cigna_vol_20d", "PG_ret_20d"], "is_new": true}, {"model_id": "new_h7_GLOBAL_XGBoost_N20_t2", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 7, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWJ_Japan_vol_20d", "TED_Spread_zscore_60d", "EWG_Germany_vol_20d", "EOG_EOGResources_ret_5d", "US7Y_Rate_ret_20d", "QQQ_vol_20d", "hmm_p_stress", "NVDA_vol_20d", "XOM_ret_1d", "CPB_CampbellSoup_ret_20d", "CTAS_Cintas_vol_20d", "EWS_Singapore_ret_5d", "TED_Spread_vol_20d", "TM_Telephone_ret_1d", "EWY_Korea_ret_20d", "EWM_Malaysia_vol_20d", "MSTR_Bitcoin3_ret_20d", "LMT_LockheedMartin_ret_1d"], "is_new": true}, {"model_id": "new_h7_GLOBAL_XGBoost_N20_t3", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 7, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "SLB_Schlumberger_ret_5d", "EWQ_France_ret_20d", "US1Y_Rate_ret_5d", "CCI_CrownCastle_vol_20d", "NVDA_vol_20d", "HangSeng_HK_ret_5d", "LOW_Lowes_ret_20d", "EWM_Malaysia_vol_20d", "PPL_PPL_ret_1d", "M_Macys_vol_20d", "ORCL_vol_20d", "CTAS_Cintas_vol_20d", "EXC_Exelon_ret_1d", "CPB_CampbellSoup_zscore_60d", "BDX_Becton_Dickinson_ret_20d", "CLX_Clorox_vol_20d", "EWQ_France_zscore_60d", "SBUX_zscore_60d"], "is_new": true}, {"model_id": "new_h7_GLOBAL_XGBoost_N20_t4", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 7, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "CCI_CrownCastle_vol_20d", "CPB_CampbellSoup_ret_5d", "NFCI_ret_5d", "SJM_JM_Smucker_ret_1d", "CPB_CampbellSoup_ret_20d", "HangSeng_HK_vol_20d", "DE_Deere_ret_5d", "EOG_EOGResources_ret_5d", "PAYX_Paychex_vol_20d", "LUV_SouthwestAir_ret_5d", "HangSeng_HK_ret_1d", "AXP_Amex_vol_20d", "US7Y_Rate_ret_20d", "INTC_ret_5d", "XLV_Health_zscore_60d", "PAYX_Paychex_zscore_60d", "INTC_ret_1d", "NEE_NextEra_ret_20d"], "is_new": true}, {"model_id": "new_h7_GLOBAL_XGBoost_N20_t5", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 7, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "AMZN_ret_5d", "NEE_NextEra_ret_20d", "MSTR_Bitcoin3_ret_1d", "PAYX_Paychex_ret_20d", "CPB_CampbellSoup_zscore_60d", "EQR_Equity_ret_1d", "GD_GeneralDynamics_zscore_60d", "XLV_Health_zscore_60d", "GE_ret_1d", "EWM_Malaysia_zscore_60d", "HangSeng_HK_ret_5d", "SLB_Schlumberger_ret_5d", "ITT_ITTInc_ret_5d", "FedFunds_zscore_60d", "DIS_vol_20d", "AORD_AUS_zscore_60d", "US3M_Rate_vol_20d", "T_ret_1d"], "is_new": true}, {"model_id": "new_h7_GLOBAL_XGBoost_N20_t6", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 7, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "SLB_Schlumberger_ret_1d", "EXC_Exelon_ret_1d", "SLB_Schlumberger_ret_5d", "EWA_Australia_zscore_60d", "TED_Spread_vol_20d", "US3M_Rate_vol_20d", "EWC_Canada_zscore_60d", "US5Y_Rate_ret_5d", "ES_Evergy_ret_1d", "HangSeng_HK_ret_5d", "EFFR_vol_20d", "spx_vol_5d", "spx_abs_ret_max_5d", "SO_SouthernCo_ret_5d", "BTI_BritishAmerican_ret_5d", "CLX_Clorox_vol_20d", "DE_Deere_vol_20d", "EWA_Australia_ret_1d"], "is_new": true}, {"model_id": "new_h7_GLOBAL_XGBoost_N20_t7", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 7, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "TXN_vol_20d", "DOW_Price_zscore_60d", "SLB_Schlumberger_ret_1d", "TED_Spread_zscore_60d", "PCAR_PaccarInc_ret_5d", "PAYX_Paychex_vol_20d", "SBUX_vol_20d", "TM_Telephone_vol_20d", "SJM_JM_Smucker_ret_1d", "EWG_Germany_ret_20d", "DAX_Germany_vol_20d", "EWM_Malaysia_ret_1d", "INTC_ret_1d", "AMD_ret_5d", "EWM_Malaysia_zscore_60d", "LMT_LockheedMartin_ret_1d", "Retail_Sales_zscore_60d", "EXC_Exelon_zscore_60d"], "is_new": true}, {"model_id": "new_h7_GLOBAL_XGBoost_N25_t0", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 7, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "hmm_p_stress", "EWA_Australia_ret_1d", "VOD_Vodafone_zscore_60d", "ORCL_zscore_60d", "TED_Spread_zscore_60d", "SLB_Schlumberger_ret_5d", "Michigan_Sentiment_ret_20d", "EWM_Malaysia_ret_1d", "3M_ret_5d", "FedFunds_zscore_60d", "AMZN_ret_5d", "PG_ret_20d", "HD_zscore_60d", "CPB_CampbellSoup_vol_20d", "VVIX_ret_20d", "SJM_JM_Smucker_ret_5d", "XLK_Tech_zscore_60d", "SLB_Schlumberger_ret_1d", "TED_Spread_vol_20d", "MS_MorganStanley_zscore_60d", "heston_var_ev_h7", "EWQ_France_zscore_60d", "CPB_CampbellSoup_ret_5d"], "is_new": true}, {"model_id": "new_h7_GLOBAL_XGBoost_N25_t1", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 7, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "HD_zscore_60d", "NFCI_ret_5d", "EWG_Germany_ret_20d", "EWS_Singapore_ret_5d", "BA_ret_1d", "EWY_Korea_ret_20d", "vix_mean_abs_ret_5d", "VRP_ma5", "GE_ret_1d", "MS_MorganStanley_ret_1d", "PG_ret_20d", "AVB_AvalonBay_zscore_60d", "DE_Deere_ret_5d", "SBUX_zscore_60d", "MSTR_Bitcoin3_ret_1d", "US1Y_Rate_ret_5d", "AMD_ret_1d", "NOC_Northrop_ret_20d", "PPL_PPL_ret_1d", "ASX_Australia_ret_5d", "FedFunds_zscore_60d", "US30Y_Rate_ret_20d", "AXP_Amex_vol_20d"], "is_new": true}, {"model_id": "new_h7_GLOBAL_XGBoost_N25_t2", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 7, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "AMD_ret_5d", "ORCL_vol_20d", "XLY_Disc_vol_20d", "AORD_AUS_zscore_60d", "EWY_Korea_ret_20d", "INTC_ret_5d", "LMT_LockheedMartin_vol_20d", "MSTR_Bitcoin3_ret_1d", "GD_GeneralDynamics_zscore_60d", "Brent_Oil_FRED_ret_5d", "MSTR_Bitcoin3_ret_5d", "PAYX_Paychex_zscore_60d", "ES_Evergy_ret_1d", "BA_ret_1d", "AMD_ret_1d", "vix_acceleration_1d", "US3M_Rate_vol_20d", "Core_PCE_zscore_60d", "DIS_vol_20d", "EOG_EOGResources_ret_5d", "CPB_CampbellSoup_vol_20d", "US30Y_Rate_ret_20d", "TED_Spread_vol_20d"], "is_new": true}, {"model_id": "new_h7_GLOBAL_XGBoost_N25_t3", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 7, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWL_Switzerland_vol_20d", "BTI_BritishAmerican_ret_20d", "AMD_ret_5d", "CI_Cigna_vol_20d", "EWY_Korea_ret_20d", "SLB_Schlumberger_ret_5d", "US1Y_Rate_ret_5d", "AXP_Amex_vol_20d", "US30Y_Rate_ret_20d", "NEE_NextEra_ret_20d", "EMR_Emerson_ret_20d", "XLB_Materials_zscore_60d", "MSTR_Bitcoin3_ret_1d", "heston_ev_h3", "EOG_EOGResources_ret_5d", "XLF_Fin_vol_20d", "DOW_Price_zscore_60d", "EWA_Australia_zscore_60d", "Core_PCE_zscore_60d", "US3Y_Rate_ret_5d", "CPB_CampbellSoup_vol_20d", "ENB_EnbridgeInc_ret_1d", "EQIX_Equinix_ret_5d"], "is_new": true}, {"model_id": "new_h7_GLOBAL_XGBoost_N25_t4", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 7, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "MRK_Merck_zscore_60d", "LLY_zscore_60d", "EWL_Switzerland_zscore_60d", "TM_Telephone_vol_20d", "EWA_Australia_zscore_60d", "PPL_PPL_ret_1d", "EWH_HongKong_ret_5d", "ITT_ITTInc_ret_5d", "Retail_Sales_zscore_60d", "VVIX_ret_20d", "XLY_Disc_vol_20d", "SLB_Schlumberger_ret_1d", "SCHW_Schwab_ret_5d", "EWQ_France_ret_20d", "HangSeng_HK_ret_1d", "MS_MorganStanley_zscore_60d", "TGT_Target_zscore_60d", "EWQ_France_zscore_60d", "CPB_CampbellSoup_ret_5d", "US1Y_Rate_ret_20d", "JNJ_ret_1d", "DHR_ret_1d", "Core_CPI_zscore_60d"], "is_new": true}, {"model_id": "new_h7_GLOBAL_XGBoost_N25_t5", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 7, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "XLF_Fin_vol_20d", "US5Y_Rate_ret_5d", "EWC_Canada_zscore_60d", "T_ret_1d", "NVDA_vol_20d", "VOD_Vodafone_zscore_60d", "EWJ_Japan_vol_20d", "US1Y_Rate_ret_20d", "EWG_Germany_vol_20d", "ASX_Australia_vol_20d", "EWS_Singapore_ret_5d", "US1Y_Rate_ret_5d", "MO_AltriaMG_ret_1d", "EXC_Exelon_zscore_60d", "PCAR_PaccarInc_ret_5d", "EFFR_ret_1d", "SPY_zscore_60d", "vix_acceleration_1d", "MRK_Merck_zscore_60d", "EWY_Korea_zscore_60d", "DHR_ret_1d", "Core_PCE_zscore_60d", "DAX_Germany_vol_20d"], "is_new": true}, {"model_id": "new_h7_GLOBAL_XGBoost_N25_t6", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 7, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "Core_PCE_zscore_60d", "US3M_Rate_vol_20d", "SBUX_vol_20d", "SLB_Schlumberger_ret_5d", "AMZN_ret_5d", "HUM_Humana_ret_5d", "US30Y_Rate_ret_20d", "CPB_CampbellSoup_ret_5d", "SJM_JM_Smucker_ret_5d", "VOD_Vodafone_zscore_60d", "AMD_ret_1d", "EFFR_ret_1d", "VRP_ma5", "AVB_AvalonBay_zscore_60d", "CPB_CampbellSoup_ret_20d", "TED_Spread_vol_20d", "CCI_CrownCastle_vol_20d", "TGT_Target_zscore_60d", "CI_Cigna_vol_20d", "EXC_Exelon_ret_1d", "ASX_Australia_vol_20d", "PFE_ret_1d", "DE_Deere_vol_20d"], "is_new": true}, {"model_id": "new_h7_GLOBAL_XGBoost_N25_t7", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 7, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "TED_Spread_vol_20d", "BTI_BritishAmerican_ret_20d", "3M_vol_20d", "ASX_Australia_vol_20d", "SBUX_zscore_60d", "AMZN_ret_5d", "LOW_Lowes_ret_5d", "spx_abs_ret_max_5d", "EWY_Korea_ret_20d", "SCHW_Schwab_ret_5d", "Retail_Sales_zscore_60d", "heston_var_ev_h3", "EWG_Germany_ret_20d", "US3Y_Rate_ret_5d", "SJM_JM_Smucker_ret_5d", "PAYX_Paychex_vol_20d", "CPB_CampbellSoup_ret_5d", "HD_ret_1d", "AVB_AvalonBay_zscore_60d", "VRP_ma5", "MS_MorganStanley_zscore_60d", "Brent_Oil_FRED_ret_20d", "vix_mean_abs_ret_5d"], "is_new": true}, {"model_id": "new_h7_GLOBAL_XGBoost_N30_t0", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 7, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "AORD_AUS_zscore_60d", "EWC_Canada_zscore_60d", "EMR_Emerson_ret_20d", "LUV_SouthwestAir_ret_5d", "HangSeng_HK_ret_1d", "HD_ret_20d", "ASX_Australia_vol_20d", "TM_Telephone_vol_20d", "Core_PCE_zscore_60d", "EWY_Korea_ret_20d", "SCHW_Schwab_ret_5d", "ENB_EnbridgeInc_ret_1d", "SBUX_vol_20d", "PPL_PPL_ret_1d", "ORCL_vol_20d", "heston_ev_h3", "CPB_CampbellSoup_ret_5d", "CPB_CampbellSoup_ret_20d", "INTC_ret_5d", "BTI_BritishAmerican_ret_20d", "XLK_Tech_zscore_60d", "NEE_NextEra_ret_20d", "MS_MorganStanley_ret_5d", "spx_momentum_3d", "CPB_CampbellSoup_vol_20d", "EWQ_France_ret_20d", "AMT_AmericanTower_ret_1d", "Nikkei_Japan_zscore_60d"], "is_new": true}, {"model_id": "new_h7_GLOBAL_XGBoost_N30_t1", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 7, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "US1Y_Rate_ret_5d", "EXC_Exelon_ret_1d", "EWM_Malaysia_vol_20d", "HD_ret_20d", "MS_MorganStanley_zscore_60d", "EWM_Malaysia_ret_1d", "Brent_Oil_FRED_ret_20d", "T10Y2Y_Spread_ret_5d", "HD_ret_5d", "PAYX_Paychex_ret_20d", "MS_MorganStanley_ret_1d", "CTAS_Cintas_vol_20d", "ENB_EnbridgeInc_ret_1d", "IBEX_Spain_ret_20d", "ASX_Australia_ret_5d", "gjr_condvar_h1", "DIS_vol_20d", "WTI_Oil_FRED_zscore_60d", "EWG_Germany_vol_20d", "PAYX_Paychex_vol_20d", "SLB_Schlumberger_ret_1d", "VOD_Vodafone_zscore_60d", "IYM_BasicMaterials_ret_20d", "HUM_Humana_ret_5d", "BA_ret_1d", "GE_ret_1d", "SBUX_vol_20d", "SBUX_zscore_60d"], "is_new": true}, {"model_id": "new_h7_GLOBAL_XGBoost_N30_t2", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 7, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "SLB_Schlumberger_ret_1d", "DE_Deere_ret_5d", "TXN_vol_20d", "HangSeng_HK_ret_5d", "CPB_CampbellSoup_vol_20d", "T10Y2Y_Spread_ret_5d", "heston_ev_h3", "US1Y_Rate_ret_5d", "US3Y_Rate_ret_5d", "GILD_Gilead_ret_20d", "NFCI_ret_5d", "US6M_Rate_ret_20d", "ITT_ITTInc_ret_5d", "T_ret_1d", "GD_GeneralDynamics_zscore_60d", "SCHW_Schwab_ret_5d", "DAX_Germany_vol_20d", "EWG_Germany_vol_20d", "Michigan_Sentiment_ret_20d", "SPY_zscore_60d", "NVDA_vol_20d", "EXC_Exelon_ret_1d", "EWA_Australia_ret_1d", "gjr_condvar_h1", "US30Y_Rate_ret_20d", "Core_CPI_zscore_60d", "DIS_vol_20d", "VVIX_ret_20d"], "is_new": true}, {"model_id": "new_h7_GLOBAL_XGBoost_N30_t3", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 7, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "vix_mean_abs_ret_5d", "Brent_Oil_FRED_ret_5d", "3M_ret_5d", "INTC_ret_5d", "EMR_Emerson_ret_20d", "CCI_CrownCastle_vol_20d", "IYM_BasicMaterials_ret_20d", "NFCI_ret_5d", "EQR_Equity_ret_1d", "SJM_JM_Smucker_ret_1d", "US1Y_Rate_ret_5d", "EWH_HongKong_ret_5d", "Brent_Oil_FRED_ret_20d", "DAX_Germany_zscore_60d", "EWM_Malaysia_vol_20d", "GILD_Gilead_ret_20d", "DIS_vol_20d", "ENB_EnbridgeInc_ret_1d", "MSTR_Bitcoin3_ret_5d", "EXC_Exelon_ret_1d", "Retail_Sales_zscore_60d", "WTI_Oil_FRED_zscore_60d", "NOC_Northrop_ret_20d", "ITT_ITTInc_ret_5d", "BA_ret_1d", "EWS_Singapore_ret_5d", "AMD_ret_5d", "BTI_BritishAmerican_ret_20d"], "is_new": true}, {"model_id": "new_h7_GLOBAL_XGBoost_N30_t4", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 7, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "heston_var_ev_h3", "SBUX_vol_20d", "EWA_Australia_ret_1d", "AORD_AUS_zscore_60d", "XLY_Disc_vol_20d", "MRK_Merck_zscore_60d", "JNJ_ret_1d", "XLK_Tech_zscore_60d", "SCHW_Schwab_ret_5d", "PAYX_Paychex_vol_20d", "XOM_ret_20d", "Core_CPI_zscore_60d", "FedFunds_zscore_60d", "EWJ_Japan_vol_20d", "Nikkei_Japan_zscore_60d", "US3M_Rate_vol_20d", "CPB_CampbellSoup_ret_20d", "PAYX_Paychex_ret_20d", "BTI_BritishAmerican_ret_5d", "ITT_ITTInc_ret_5d", "CI_Cigna_vol_20d", "DE_Deere_ret_5d", "BDX_Becton_Dickinson_ret_20d", "HD_ret_20d", "MSTR_Bitcoin3_ret_20d", "HangSeng_HK_vol_20d", "HUM_Humana_ret_5d", "US3M_Rate_zscore_60d"], "is_new": true}, {"model_id": "new_h7_GLOBAL_XGBoost_N30_t5", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 7, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "BTI_BritishAmerican_ret_5d", "EWJ_Japan_vol_20d", "XLF_Fin_vol_20d", "SBUX_vol_20d", "HangSeng_HK_ret_5d", "NVDA_vol_20d", "Michigan_Sentiment_ret_20d", "XLV_Health_zscore_60d", "US3M_Rate_zscore_60d", "EWM_Malaysia_zscore_60d", "spx_abs_ret_max_5d", "Nikkei_Japan_zscore_60d", "HangSeng_HK_ret_1d", "VVIX_ret_20d", "BLK_BlackRock_zscore_60d", "LOW_Lowes_ret_20d", "PG_ret_20d", "INTC_ret_1d", "EMR_Emerson_ret_20d", "3M_ret_5d", "PCAR_PaccarInc_ret_5d", "EOG_EOGResources_vol_20d", "IBEX_Spain_ret_20d", "US1Y_Rate_ret_5d", "NOC_Northrop_ret_20d", "EWG_Germany_ret_20d", "CPB_CampbellSoup_zscore_60d", "vix_mean_abs_ret_5d"], "is_new": true}, {"model_id": "new_h7_GLOBAL_XGBoost_N30_t6", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 7, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "heston_var_ev_h3", "US30Y_Rate_ret_20d", "CMCSA_ret_1d", "EWA_Australia_ret_1d", "EFFR_ret_1d", "AMD_ret_1d", "CPB_CampbellSoup_zscore_60d", "Nikkei_Japan_zscore_60d", "AMT_AmericanTower_ret_1d", "QQQ_vol_20d", "MSTR_Bitcoin3_ret_20d", "DIS_vol_20d", "MSTR_Bitcoin3_ret_5d", "Industrial_Production_zscore_60d", "AORD_AUS_zscore_60d", "Nikkei_Japan_vol_20d", "IBEX_Spain_ret_20d", "XLV_Health_zscore_60d", "EWJ_Japan_vol_20d", "NEE_NextEra_ret_20d", "TM_Telephone_vol_20d", "HangSeng_HK_ret_1d", "3M_ret_5d", "Michigan_Sentiment_ret_20d", "PAYX_Paychex_zscore_60d", "EMR_Emerson_ret_20d", "EFFR_vol_20d", "LOW_Lowes_ret_20d"], "is_new": true}, {"model_id": "new_h7_GLOBAL_XGBoost_N30_t7", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 7, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "heston_var_ev_h5", "BTI_BritishAmerican_ret_5d", "DAX_Germany_zscore_60d", "CPB_CampbellSoup_zscore_60d", "ASX_Australia_vol_20d", "PAYX_Paychex_zscore_60d", "ES_Evergy_ret_1d", "MSTR_Bitcoin3_ret_1d", "HangSeng_HK_vol_20d", "DOW_Price_zscore_60d", "SBUX_zscore_60d", "heston_var_ev_h7", "ITT_ITTInc_ret_5d", "BTI_BritishAmerican_ret_20d", "M_Macys_vol_20d", "spx_abs_ret_max_5d", "VVIX_ret_20d", "NVDA_vol_20d", "EXC_Exelon_zscore_60d", "PG_ret_20d", "gjr_condvar_h1", "CI_Cigna_vol_20d", "Michigan_Sentiment_ret_20d", "IWM_SmallCap_vol_20d", "EWH_HongKong_ret_5d", "LMT_LockheedMartin_ret_1d", "GILD_Gilead_ret_20d", "EWQ_France_zscore_60d"], "is_new": true}, {"model_id": "new_h7_GLOBAL_LightGBM_N5_t0", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 7, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "IWM_SmallCap_vol_20d", "SBUX_vol_20d", "BTI_BritishAmerican_ret_20d"], "is_new": true}, {"model_id": "new_h7_GLOBAL_LightGBM_N5_t1", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 7, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "SBUX_ret_5d", "PFE_ret_1d", "ASX_Australia_ret_5d"], "is_new": true}, {"model_id": "new_h7_GLOBAL_LightGBM_N5_t2", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 7, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "ES_Evergy_ret_1d", "XLK_Tech_zscore_60d", "DOW_Price_zscore_60d"], "is_new": true}, {"model_id": "new_h7_GLOBAL_LightGBM_N5_t3", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 7, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWG_Germany_vol_20d", "LMT_LockheedMartin_ret_1d", "Core_CPI_zscore_60d"], "is_new": true}, {"model_id": "new_h7_GLOBAL_LightGBM_N5_t4", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 7, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "HangSeng_HK_ret_1d", "HUM_Humana_ret_5d", "EWS_Singapore_ret_5d"], "is_new": true}, {"model_id": "new_h7_GLOBAL_LightGBM_N5_t5", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 7, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWS_Singapore_ret_5d", "DHR_ret_1d", "SPY_zscore_60d"], "is_new": true}, {"model_id": "new_h7_GLOBAL_LightGBM_N5_t6", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 7, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "MSTR_Bitcoin3_ret_20d", "T_ret_1d", "TED_Spread_zscore_60d"], "is_new": true}, {"model_id": "new_h7_GLOBAL_LightGBM_N5_t7", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 7, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "SO_SouthernCo_ret_5d", "NEE_NextEra_ret_20d", "vix_acceleration_1d"], "is_new": true}, {"model_id": "new_h7_GLOBAL_LightGBM_N8_t0", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 7, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "BTI_BritishAmerican_ret_5d", "EWL_Switzerland_zscore_60d", "NWL_Newell_ret_20d", "AXP_Amex_vol_20d", "SJM_JM_Smucker_ret_5d", "AVB_AvalonBay_zscore_60d"], "is_new": true}, {"model_id": "new_h7_GLOBAL_LightGBM_N8_t1", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 7, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "QQQ_vol_20d", "GE_ret_1d", "heston_ev_h3", "DHR_ret_1d", "SBUX_vol_20d", "EXC_Exelon_zscore_60d"], "is_new": true}, {"model_id": "new_h7_GLOBAL_LightGBM_N8_t2", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 7, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "hmm_p_stress", "HangSeng_HK_vol_20d", "SJM_JM_Smucker_ret_5d", "HD_zscore_60d", "EOG_EOGResources_ret_5d", "NOC_Northrop_ret_20d"], "is_new": true}, {"model_id": "new_h7_GLOBAL_LightGBM_N8_t3", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 7, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "Brent_Oil_FRED_ret_5d", "DAX_Germany_vol_20d", "US30Y_Rate_ret_20d", "ITT_ITTInc_ret_5d", "LOW_Lowes_ret_5d", "M_Macys_vol_20d"], "is_new": true}, {"model_id": "new_h7_GLOBAL_LightGBM_N8_t4", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 7, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "AMT_AmericanTower_ret_1d", "US1Y_Rate_ret_20d", "EMR_Emerson_ret_20d", "T10Y2Y_Spread_ret_5d", "SPY_zscore_60d", "EWQ_France_ret_20d"], "is_new": true}, {"model_id": "new_h7_GLOBAL_LightGBM_N8_t5", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 7, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "MSTR_Bitcoin3_ret_1d", "XLB_Materials_zscore_60d", "LLY_zscore_60d", "IWM_SmallCap_vol_20d", "EWY_Korea_zscore_60d", "LOW_Lowes_ret_20d"], "is_new": true}, {"model_id": "new_h7_GLOBAL_LightGBM_N8_t6", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 7, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "INTC_ret_5d", "EFFR_ret_1d", "IWM_SmallCap_vol_20d", "EQR_Equity_ret_1d", "NVDA_vol_20d", "Brent_Oil_FRED_ret_5d"], "is_new": true}, {"model_id": "new_h7_GLOBAL_LightGBM_N8_t7", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 7, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "BA_ret_1d", "INTC_ret_5d", "heston_var_ev_h5", "Nikkei_Japan_vol_20d", "DE_Deere_vol_20d", "JNJ_ret_1d"], "is_new": true}, {"model_id": "new_h7_GLOBAL_LightGBM_N10_t0", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 7, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWA_Australia_zscore_60d", "gjr_condvar_h1", "MSTR_Bitcoin3_ret_5d", "BA_ret_1d", "EOG_EOGResources_ret_5d", "NEE_NextEra_ret_20d", "INTC_ret_5d", "DAX_Germany_vol_20d"], "is_new": true}, {"model_id": "new_h7_GLOBAL_LightGBM_N10_t1", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 7, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "heston_var_ev_h5", "EWG_Germany_vol_20d", "Retail_Sales_zscore_60d", "EWA_Australia_zscore_60d", "spx_vol_5d", "DE_Deere_ret_5d", "EWH_HongKong_ret_5d", "HD_zscore_60d"], "is_new": true}, {"model_id": "new_h7_GLOBAL_LightGBM_N10_t2", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 7, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EXC_Exelon_ret_1d", "SCHW_Schwab_ret_5d", "NVDA_vol_20d", "M_Macys_vol_20d", "XLV_Health_zscore_60d", "spx_abs_ret_max_5d", "INTC_ret_1d", "XLB_Materials_zscore_60d"], "is_new": true}, {"model_id": "new_h7_GLOBAL_LightGBM_N10_t3", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 7, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWM_Malaysia_vol_20d", "AVB_AvalonBay_zscore_60d", "US3Y_Rate_ret_5d", "AXP_Amex_ret_20d", "XOM_ret_20d", "IWM_SmallCap_vol_20d", "ENB_EnbridgeInc_ret_1d", "EWL_Switzerland_vol_20d"], "is_new": true}, {"model_id": "new_h7_GLOBAL_LightGBM_N10_t4", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 7, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EXC_Exelon_ret_1d", "IYR_US_REIT2_zscore_60d", "GE_ret_1d", "gjr_condvar_h1", "NFCI_ret_5d", "CI_Cigna_vol_20d", "LOW_Lowes_ret_5d", "EQR_Equity_ret_1d"], "is_new": true}, {"model_id": "new_h7_GLOBAL_LightGBM_N10_t5", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 7, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "CPB_CampbellSoup_vol_20d", "US6M_Rate_ret_20d", "spx_abs_ret_max_5d", "MS_MorganStanley_ret_5d", "SJM_JM_Smucker_ret_1d", "INTC_ret_1d", "SBUX_zscore_60d", "SCHW_Schwab_ret_5d"], "is_new": true}, {"model_id": "new_h7_GLOBAL_LightGBM_N10_t6", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 7, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "SJM_JM_Smucker_ret_5d", "BLK_BlackRock_zscore_60d", "GILD_Gilead_ret_20d", "EMR_Emerson_ret_20d", "heston_var_ev_h3", "Core_CPI_zscore_60d", "AMD_ret_1d", "AMGN_Amgen_ret_1d"], "is_new": true}, {"model_id": "new_h7_GLOBAL_LightGBM_N10_t7", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 7, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "DAX_Germany_vol_20d", "T_ret_1d", "EOG_EOGResources_vol_20d", "EWH_HongKong_ret_5d", "MO_AltriaMG_ret_1d", "TED_Spread_vol_20d", "NWL_Newell_ret_20d", "EWG_Germany_vol_20d"], "is_new": true}, {"model_id": "new_h7_GLOBAL_LightGBM_N12_t0", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 7, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "CLX_Clorox_vol_20d", "heston_var_ev_h5", "GD_GeneralDynamics_zscore_60d", "T10Y2Y_Spread_ret_5d", "WTI_Oil_FRED_zscore_60d", "BLK_BlackRock_zscore_60d", "HangSeng_HK_ret_5d", "HD_ret_1d", "US3M_Rate_zscore_60d", "Industrial_Production_zscore_60d"], "is_new": true}, {"model_id": "new_h7_GLOBAL_LightGBM_N12_t1", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 7, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "GILD_Gilead_ret_20d", "LUV_SouthwestAir_ret_5d", "PLD_Prologis_ret_5d", "BDX_Becton_Dickinson_ret_20d", "spx_vol_5d", "DOW_Price_zscore_60d", "NEE_NextEra_ret_20d", "spx_momentum_3d", "XLB_Materials_zscore_60d", "Core_PCE_zscore_60d"], "is_new": true}, {"model_id": "new_h7_GLOBAL_LightGBM_N12_t2", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 7, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "MRK_Merck_zscore_60d", "FedFunds_zscore_60d", "CPB_CampbellSoup_vol_20d", "DOW_Price_zscore_60d", "SBUX_vol_20d", "EWL_Switzerland_zscore_60d", "EWH_HongKong_ret_5d", "HD_ret_5d", "LUV_SouthwestAir_ret_5d", "AXP_Amex_vol_20d"], "is_new": true}, {"model_id": "new_h7_GLOBAL_LightGBM_N12_t3", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 7, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "LOW_Lowes_ret_5d", "EWL_Switzerland_vol_20d", "heston_var_ev_h7", "TED_Spread_zscore_60d", "HUM_Humana_ret_5d", "Retail_Sales_zscore_60d", "NOC_Northrop_ret_20d", "EOG_EOGResources_vol_20d", "EWJ_Japan_vol_20d", "MS_MorganStanley_ret_5d"], "is_new": true}, {"model_id": "new_h7_GLOBAL_LightGBM_N12_t4", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 7, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "spx_abs_ret_max_5d", "BDX_Becton_Dickinson_ret_20d", "PG_ret_20d", "DAX_Germany_vol_20d", "EWM_Malaysia_zscore_60d", "INTC_ret_5d", "PPL_PPL_ret_1d", "WTI_Oil_FRED_zscore_60d", "spx_momentum_3d", "MSTR_Bitcoin3_ret_20d"], "is_new": true}, {"model_id": "new_h7_GLOBAL_LightGBM_N12_t5", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 7, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "MS_MorganStanley_ret_5d", "XLB_Materials_zscore_60d", "LOW_Lowes_ret_20d", "spx_momentum_3d", "CPB_CampbellSoup_ret_5d", "EWM_Malaysia_vol_20d", "US7Y_Rate_ret_20d", "HangSeng_HK_ret_1d", "PLD_Prologis_ret_5d", "ITT_ITTInc_ret_5d"], "is_new": true}, {"model_id": "new_h7_GLOBAL_LightGBM_N12_t6", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 7, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "XLK_Tech_zscore_60d", "MSTR_Bitcoin3_ret_5d", "PAYX_Paychex_zscore_60d", "PFE_ret_1d", "US30Y_Rate_ret_20d", "PPL_PPL_ret_1d", "heston_var_ev_h5", "EWM_Malaysia_vol_20d", "PCAR_PaccarInc_ret_5d", "BTI_BritishAmerican_ret_5d"], "is_new": true}, {"model_id": "new_h7_GLOBAL_LightGBM_N12_t7", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 7, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWC_Canada_zscore_60d", "EWL_Switzerland_zscore_60d", "BDX_Becton_Dickinson_ret_20d", "hmm_p_stress", "BA_ret_1d", "Nikkei_Japan_vol_20d", "DHR_ret_1d", "Retail_Sales_zscore_60d", "ORCL_zscore_60d", "DHR_vol_20d"], "is_new": true}, {"model_id": "new_h7_GLOBAL_LightGBM_N15_t0", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 7, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "gjr_condvar_h1", "EWS_Singapore_ret_5d", "IWM_SmallCap_vol_20d", "heston_ev_h3", "M_Macys_vol_20d", "VRP_ma5", "GILD_Gilead_ret_20d", "SBUX_vol_20d", "BTI_BritishAmerican_ret_5d", "EOG_EOGResources_ret_5d", "Nikkei_Japan_zscore_60d", "EWM_Malaysia_ret_1d", "NVDA_vol_20d"], "is_new": true}, {"model_id": "new_h7_GLOBAL_LightGBM_N15_t1", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 7, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWS_Singapore_ret_5d", "Core_PCE_zscore_60d", "EWY_Korea_zscore_60d", "AORD_AUS_zscore_60d", "QQQ_vol_20d", "EXC_Exelon_ret_1d", "XLB_Materials_zscore_60d", "AMD_ret_1d", "3M_vol_20d", "Nikkei_Japan_vol_20d", "SLB_Schlumberger_ret_1d", "EWY_Korea_ret_20d", "vix_mean_abs_ret_5d"], "is_new": true}, {"model_id": "new_h7_GLOBAL_LightGBM_N15_t2", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 7, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EQIX_Equinix_ret_5d", "spx_vol_5d", "CPB_CampbellSoup_zscore_60d", "ENB_EnbridgeInc_ret_1d", "EWL_Switzerland_zscore_60d", "AMT_AmericanTower_ret_1d", "US7Y_Rate_ret_20d", "AMD_ret_5d", "TED_Spread_zscore_60d", "Nikkei_Japan_vol_20d", "CTAS_Cintas_vol_20d", "EFFR_vol_20d", "AMGN_Amgen_ret_1d"], "is_new": true}, {"model_id": "new_h7_GLOBAL_LightGBM_N15_t3", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 7, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "MSTR_Bitcoin3_ret_5d", "IYM_BasicMaterials_ret_20d", "MS_MorganStanley_ret_5d", "vix_mean_abs_ret_5d", "EWQ_France_ret_20d", "EWM_Malaysia_vol_20d", "LUV_SouthwestAir_ret_5d", "EWG_Germany_ret_20d", "MSTR_Bitcoin3_ret_20d", "TED_Spread_zscore_60d", "EWC_Canada_zscore_60d", "AVB_AvalonBay_zscore_60d", "heston_var_ev_h7"], "is_new": true}, {"model_id": "new_h7_GLOBAL_LightGBM_N15_t4", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 7, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "INTC_ret_1d", "SBUX_zscore_60d", "DHR_ret_1d", "EWL_Switzerland_zscore_60d", "Nikkei_Japan_zscore_60d", "HUM_Humana_ret_5d", "3M_ret_5d", "EWC_Canada_zscore_60d", "AMZN_ret_5d", "MSTR_Bitcoin3_ret_5d", "ITT_ITTInc_ret_5d", "BLK_BlackRock_zscore_60d", "XLY_Disc_vol_20d"], "is_new": true}, {"model_id": "new_h7_GLOBAL_LightGBM_N15_t5", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 7, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "INTC_ret_5d", "spx_abs_ret_max_5d", "IBEX_Spain_ret_20d", "vix_acceleration_1d", "XOM_ret_1d", "AMD_ret_5d", "HD_zscore_60d", "SBUX_zscore_60d", "MSTR_Bitcoin3_ret_5d", "heston_ev_h3", "Nikkei_Japan_zscore_60d", "EWL_Switzerland_zscore_60d", "VRP_ma5"], "is_new": true}, {"model_id": "new_h7_GLOBAL_LightGBM_N15_t6", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 7, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "DHR_ret_1d", "US6M_Rate_ret_20d", "IYR_US_REIT2_zscore_60d", "EQIX_Equinix_ret_5d", "MS_MorganStanley_ret_5d", "EWA_Australia_zscore_60d", "MO_AltriaMG_ret_1d", "VVIX_ret_20d", "BA_ret_1d", "HD_ret_1d", "EWG_Germany_ret_20d", "GILD_Gilead_ret_20d", "DE_Deere_ret_5d"], "is_new": true}, {"model_id": "new_h7_GLOBAL_LightGBM_N15_t7", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 7, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "PPL_PPL_ret_1d", "MS_MorganStanley_ret_5d", "MS_MorganStanley_ret_1d", "Brent_Oil_FRED_ret_20d", "EWS_Singapore_ret_5d", "3M_ret_5d", "EWM_Malaysia_ret_1d", "AMGN_Amgen_ret_1d", "NEE_NextEra_ret_20d", "hmm_p_stress", "3M_vol_20d", "CPB_CampbellSoup_zscore_60d", "VRP_ma5"], "is_new": true}, {"model_id": "new_h7_GLOBAL_LightGBM_N20_t0", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 7, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWA_Australia_zscore_60d", "CMCSA_ret_1d", "TGT_Target_zscore_60d", "T_ret_1d", "HD_ret_1d", "DHR_vol_20d", "GD_GeneralDynamics_zscore_60d", "SCHW_Schwab_ret_5d", "TM_Telephone_ret_1d", "WTI_Oil_FRED_zscore_60d", "EWC_Canada_zscore_60d", "BLK_BlackRock_zscore_60d", "MSTR_Bitcoin3_ret_5d", "US3M_Rate_vol_20d", "DIS_vol_20d", "XLV_Health_zscore_60d", "AMT_AmericanTower_ret_1d", "JNJ_ret_1d"], "is_new": true}, {"model_id": "new_h7_GLOBAL_LightGBM_N20_t1", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 7, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "SPY_zscore_60d", "Brent_Oil_FRED_ret_20d", "EFFR_vol_20d", "US30Y_Rate_ret_20d", "LOW_Lowes_ret_20d", "TM_Telephone_vol_20d", "ES_Evergy_ret_1d", "VRP_ma5", "GE_ret_1d", "EWJ_Japan_vol_20d", "ASX_Australia_vol_20d", "AMD_ret_5d", "IBEX_Spain_ret_20d", "CPB_CampbellSoup_ret_20d", "EOG_EOGResources_ret_5d", "Michigan_Sentiment_ret_20d", "ASX_Australia_ret_5d", "EWH_HongKong_ret_5d"], "is_new": true}, {"model_id": "new_h7_GLOBAL_LightGBM_N20_t2", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 7, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "heston_var_ev_h3", "gjr_condvar_h1", "US30Y_Rate_ret_20d", "LOW_Lowes_ret_20d", "XOM_ret_1d", "INTC_ret_1d", "SLB_Schlumberger_ret_5d", "ASX_Australia_vol_20d", "Brent_Oil_FRED_ret_20d", "EWA_Australia_ret_1d", "DAX_Germany_zscore_60d", "MS_MorganStanley_ret_1d", "GE_ret_1d", "HangSeng_HK_ret_1d", "SBUX_vol_20d", "EWL_Switzerland_vol_20d", "US7Y_Rate_ret_20d", "T_ret_1d"], "is_new": true}, {"model_id": "new_h7_GLOBAL_LightGBM_N20_t3", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 7, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "spx_abs_ret_max_5d", "HD_ret_5d", "TED_Spread_zscore_60d", "NFCI_ret_5d", "gjr_condvar_h1", "LOW_Lowes_ret_5d", "EWJ_Japan_vol_20d", "CPB_CampbellSoup_ret_20d", "EOG_EOGResources_vol_20d", "spx_momentum_3d", "US1Y_Rate_ret_20d", "heston_var_ev_h7", "MO_AltriaMG_ret_1d", "MRK_Merck_zscore_60d", "EOG_EOGResources_ret_5d", "PAYX_Paychex_zscore_60d", "WTI_Oil_FRED_zscore_60d", "XLB_Materials_zscore_60d"], "is_new": true}, {"model_id": "new_h7_GLOBAL_LightGBM_N20_t4", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 7, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "LLY_zscore_60d", "HangSeng_HK_ret_5d", "SJM_JM_Smucker_ret_5d", "heston_var_ev_h7", "vix_acceleration_1d", "3M_vol_20d", "Core_CPI_zscore_60d", "gjr_condvar_h1", "BDX_Becton_Dickinson_ret_20d", "EWM_Malaysia_vol_20d", "CI_Cigna_vol_20d", "PCAR_PaccarInc_ret_5d", "EWY_Korea_zscore_60d", "spx_abs_ret_max_5d", "SBUX_zscore_60d", "AORD_AUS_zscore_60d", "HD_ret_20d", "CMCSA_ret_1d"], "is_new": true}, {"model_id": "new_h7_GLOBAL_LightGBM_N20_t5", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 7, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWS_Singapore_ret_5d", "AMGN_Amgen_ret_1d", "PCAR_PaccarInc_ret_5d", "SJM_JM_Smucker_ret_5d", "BTI_BritishAmerican_ret_20d", "TED_Spread_vol_20d", "AMZN_ret_5d", "MSTR_Bitcoin3_ret_20d", "JNJ_ret_1d", "VOD_Vodafone_zscore_60d", "EWM_Malaysia_vol_20d", "EWL_Switzerland_vol_20d", "XOM_ret_20d", "TED_Spread_zscore_60d", "CMCSA_ret_1d", "HangSeng_HK_ret_5d", "spx_vol_5d", "EWQ_France_zscore_60d"], "is_new": true}, {"model_id": "new_h7_GLOBAL_LightGBM_N20_t6", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 7, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "LLY_zscore_60d", "CCI_CrownCastle_vol_20d", "JNJ_ret_1d", "LUV_SouthwestAir_ret_5d", "DIS_vol_20d", "EXC_Exelon_ret_1d", "CLX_Clorox_vol_20d", "LMT_LockheedMartin_ret_1d", "SBUX_ret_5d", "PAYX_Paychex_vol_20d", "VVIX_ret_20d", "EOG_EOGResources_ret_5d", "heston_var_ev_h3", "IYM_BasicMaterials_ret_20d", "TM_Telephone_vol_20d", "XOM_ret_20d", "INTC_ret_5d", "EWL_Switzerland_vol_20d"], "is_new": true}, {"model_id": "new_h7_GLOBAL_LightGBM_N20_t7", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 7, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "MSTR_Bitcoin3_ret_20d", "LUV_SouthwestAir_ret_5d", "heston_var_ev_h5", "SPY_zscore_60d", "DE_Deere_ret_5d", "gjr_condvar_h1", "hmm_p_stress", "EWQ_France_ret_20d", "spx_momentum_3d", "SJM_JM_Smucker_ret_5d", "CTAS_Cintas_vol_20d", "GD_GeneralDynamics_zscore_60d", "HD_ret_20d", "VVIX_ret_20d", "EWA_Australia_ret_1d", "ES_Evergy_ret_1d", "US1Y_Rate_ret_20d", "XLB_Materials_zscore_60d"], "is_new": true}, {"model_id": "new_h7_GLOBAL_LightGBM_N25_t0", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 7, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "SPY_zscore_60d", "BDX_Becton_Dickinson_ret_20d", "IYM_BasicMaterials_ret_20d", "EWA_Australia_ret_1d", "EWG_Germany_vol_20d", "PPL_PPL_ret_1d", "US30Y_Rate_ret_20d", "INTC_ret_1d", "Core_CPI_zscore_60d", "ENB_EnbridgeInc_ret_1d", "EFFR_ret_1d", "EOG_EOGResources_vol_20d", "US3Y_Rate_ret_5d", "LUV_SouthwestAir_ret_5d", "EWL_Switzerland_vol_20d", "IYR_US_REIT2_zscore_60d", "Nikkei_Japan_zscore_60d", "GE_ret_1d", "NFCI_ret_5d", "BLK_BlackRock_zscore_60d", "AMD_ret_5d", "LOW_Lowes_ret_5d", "XLF_Fin_vol_20d"], "is_new": true}, {"model_id": "new_h7_GLOBAL_LightGBM_N25_t1", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 7, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "GD_GeneralDynamics_zscore_60d", "US30Y_Rate_ret_20d", "SLB_Schlumberger_ret_1d", "MSTR_Bitcoin3_ret_20d", "spx_vol_5d", "US6M_Rate_ret_20d", "M_Macys_vol_20d", "JNJ_ret_1d", "ORCL_zscore_60d", "BTI_BritishAmerican_ret_20d", "FedFunds_zscore_60d", "NVDA_vol_20d", "SJM_JM_Smucker_ret_5d", "GILD_Gilead_ret_20d", "SBUX_zscore_60d", "EQR_Equity_ret_1d", "XOM_ret_1d", "VRP_ma5", "MS_MorganStanley_zscore_60d", "EWM_Malaysia_ret_1d", "3M_ret_5d", "TXN_vol_20d", "DAX_Germany_vol_20d"], "is_new": true}, {"model_id": "new_h7_GLOBAL_LightGBM_N25_t2", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 7, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "heston_ev_h3", "EWS_Singapore_ret_5d", "gjr_condvar_h1", "XLF_Fin_vol_20d", "SBUX_ret_5d", "DAX_Germany_vol_20d", "DIS_vol_20d", "XOM_ret_20d", "MSTR_Bitcoin3_ret_20d", "EWJ_Japan_vol_20d", "Retail_Sales_zscore_60d", "ORCL_zscore_60d", "ES_Evergy_ret_1d", "spx_vol_5d", "EWG_Germany_ret_20d", "AMD_ret_1d", "CPB_CampbellSoup_vol_20d", "HD_zscore_60d", "AORD_AUS_zscore_60d", "T10Y2Y_Spread_ret_5d", "LUV_SouthwestAir_ret_5d", "WTI_Oil_FRED_zscore_60d", "SJM_JM_Smucker_ret_1d"], "is_new": true}, {"model_id": "new_h7_GLOBAL_LightGBM_N25_t3", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 7, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "Michigan_Sentiment_ret_20d", "PAYX_Paychex_ret_20d", "JNJ_ret_1d", "CCI_CrownCastle_vol_20d", "M_Macys_vol_20d", "Brent_Oil_FRED_ret_5d", "AORD_AUS_zscore_60d", "DAX_Germany_zscore_60d", "SBUX_vol_20d", "ORCL_zscore_60d", "US6M_Rate_ret_20d", "US30Y_Rate_ret_20d", "SLB_Schlumberger_ret_5d", "spx_vol_5d", "EWQ_France_zscore_60d", "EWL_Switzerland_zscore_60d", "VRP_ma5", "Nikkei_Japan_zscore_60d", "CMCSA_ret_1d", "EWG_Germany_ret_20d", "SO_SouthernCo_ret_5d", "CI_Cigna_vol_20d", "AXP_Amex_ret_20d"], "is_new": true}, {"model_id": "new_h7_GLOBAL_LightGBM_N25_t4", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 7, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "PCAR_PaccarInc_ret_5d", "XLB_Materials_zscore_60d", "NOC_Northrop_ret_20d", "CTAS_Cintas_vol_20d", "DHR_ret_1d", "AMD_ret_1d", "SBUX_zscore_60d", "US3Y_Rate_ret_5d", "LOW_Lowes_ret_5d", "US3M_Rate_zscore_60d", "heston_var_ev_h7", "AMGN_Amgen_ret_1d", "XOM_ret_1d", "EWA_Australia_zscore_60d", "EFFR_vol_20d", "CPB_CampbellSoup_vol_20d", "GE_ret_1d", "spx_momentum_3d", "T_ret_1d", "CI_Cigna_vol_20d", "TM_Telephone_vol_20d", "HUM_Humana_ret_5d", "EWM_Malaysia_vol_20d"], "is_new": true}, {"model_id": "new_h7_GLOBAL_LightGBM_N25_t5", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 7, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "SCHW_Schwab_ret_5d", "HD_ret_20d", "AMT_AmericanTower_ret_1d", "EWS_Singapore_ret_5d", "XOM_ret_1d", "PAYX_Paychex_ret_20d", "ES_Evergy_ret_1d", "heston_var_ev_h7", "EWY_Korea_ret_20d", "HD_ret_1d", "EOG_EOGResources_vol_20d", "CMCSA_ret_1d", "INTC_ret_5d", "HangSeng_HK_ret_5d", "EWL_Switzerland_vol_20d", "TM_Telephone_vol_20d", "ASX_Australia_ret_5d", "AMGN_Amgen_ret_1d", "PCAR_PaccarInc_ret_5d", "DE_Deere_ret_5d", "US3M_Rate_zscore_60d", "DHR_ret_1d", "EWH_HongKong_ret_5d"], "is_new": true}, {"model_id": "new_h7_GLOBAL_LightGBM_N25_t6", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 7, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "HD_ret_20d", "Nikkei_Japan_zscore_60d", "US30Y_Rate_ret_20d", "Industrial_Production_zscore_60d", "vix_acceleration_1d", "EXC_Exelon_ret_1d", "WTI_Oil_FRED_zscore_60d", "ES_Evergy_ret_1d", "EFFR_ret_1d", "Nikkei_Japan_vol_20d", "BLK_BlackRock_zscore_60d", "EWY_Korea_zscore_60d", "EOG_EOGResources_vol_20d", "CPB_CampbellSoup_zscore_60d", "SBUX_ret_5d", "heston_var_ev_h3", "LOW_Lowes_ret_5d", "GILD_Gilead_ret_20d", "ASX_Australia_vol_20d", "CPB_CampbellSoup_ret_20d", "hmm_p_stress", "spx_vol_5d", "spx_abs_ret_max_5d"], "is_new": true}, {"model_id": "new_h7_GLOBAL_LightGBM_N25_t7", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 7, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWQ_France_ret_20d", "VVIX_ret_20d", "AMZN_ret_5d", "IYM_BasicMaterials_ret_20d", "EWA_Australia_zscore_60d", "NFCI_ret_5d", "DE_Deere_ret_5d", "Nikkei_Japan_zscore_60d", "Brent_Oil_FRED_ret_20d", "ORCL_zscore_60d", "SJM_JM_Smucker_ret_1d", "CI_Cigna_vol_20d", "EWM_Malaysia_zscore_60d", "IWM_SmallCap_vol_20d", "EWG_Germany_vol_20d", "SPY_zscore_60d", "US7Y_Rate_ret_20d", "BDX_Becton_Dickinson_ret_20d", "US3M_Rate_zscore_60d", "MS_MorganStanley_ret_1d", "VOD_Vodafone_zscore_60d", "LMT_LockheedMartin_vol_20d", "US3Y_Rate_ret_5d"], "is_new": true}, {"model_id": "new_h7_GLOBAL_LightGBM_N30_t0", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 7, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "US3M_Rate_zscore_60d", "EFFR_ret_1d", "HangSeng_HK_ret_5d", "AMD_ret_1d", "ITT_ITTInc_ret_5d", "US30Y_Rate_ret_20d", "BA_ret_1d", "PAYX_Paychex_ret_20d", "EWQ_France_zscore_60d", "NWL_Newell_ret_20d", "LMT_LockheedMartin_vol_20d", "MS_MorganStanley_ret_5d", "CCI_CrownCastle_vol_20d", "CI_Cigna_vol_20d", "SO_SouthernCo_ret_5d", "PAYX_Paychex_zscore_60d", "EXC_Exelon_zscore_60d", "AXP_Amex_vol_20d", "Retail_Sales_zscore_60d", "XLF_Fin_vol_20d", "VOD_Vodafone_zscore_60d", "Brent_Oil_FRED_ret_20d", "Core_PCE_zscore_60d", "LOW_Lowes_ret_5d", "LOW_Lowes_ret_20d", "EWC_Canada_zscore_60d", "AORD_AUS_zscore_60d", "DAX_Germany_vol_20d"], "is_new": true}, {"model_id": "new_h7_GLOBAL_LightGBM_N30_t1", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 7, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "HD_zscore_60d", "EWM_Malaysia_vol_20d", "IWM_SmallCap_vol_20d", "AORD_AUS_zscore_60d", "heston_var_ev_h7", "PPL_PPL_ret_1d", "AXP_Amex_ret_20d", "Core_CPI_zscore_60d", "XOM_ret_20d", "XLY_Disc_vol_20d", "EFFR_vol_20d", "US6M_Rate_ret_20d", "MRK_Merck_zscore_60d", "3M_ret_5d", "LMT_LockheedMartin_vol_20d", "AVB_AvalonBay_zscore_60d", "SPY_zscore_60d", "NVDA_vol_20d", "MS_MorganStanley_ret_1d", "CCI_CrownCastle_vol_20d", "3M_vol_20d", "CPB_CampbellSoup_zscore_60d", "Nikkei_Japan_zscore_60d", "LOW_Lowes_ret_5d", "ASX_Australia_vol_20d", "HD_ret_20d", "EWC_Canada_zscore_60d", "TED_Spread_vol_20d"], "is_new": true}, {"model_id": "new_h7_GLOBAL_LightGBM_N30_t2", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 7, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "HangSeng_HK_vol_20d", "ENB_EnbridgeInc_ret_1d", "CLX_Clorox_vol_20d", "TED_Spread_vol_20d", "INTC_ret_5d", "Core_CPI_zscore_60d", "DAX_Germany_zscore_60d", "BA_ret_1d", "EWY_Korea_ret_20d", "MSTR_Bitcoin3_ret_5d", "DIS_vol_20d", "NWL_Newell_ret_20d", "IYR_US_REIT2_zscore_60d", "EWL_Switzerland_zscore_60d", "EWG_Germany_vol_20d", "MSTR_Bitcoin3_ret_1d", "XOM_ret_1d", "XOM_ret_20d", "HD_zscore_60d", "EWS_Singapore_ret_5d", "Nikkei_Japan_zscore_60d", "US3M_Rate_zscore_60d", "EWY_Korea_zscore_60d", "EQIX_Equinix_ret_5d", "3M_ret_5d", "PAYX_Paychex_vol_20d", "MO_AltriaMG_ret_1d", "HD_ret_20d"], "is_new": true}, {"model_id": "new_h7_GLOBAL_LightGBM_N30_t3", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 7, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWS_Singapore_ret_5d", "HangSeng_HK_ret_1d", "US3M_Rate_zscore_60d", "vix_mean_abs_ret_5d", "DE_Deere_ret_5d", "HD_ret_1d", "CI_Cigna_vol_20d", "IWM_SmallCap_vol_20d", "MS_MorganStanley_ret_5d", "SJM_JM_Smucker_ret_5d", "US1Y_Rate_ret_20d", "AVB_AvalonBay_zscore_60d", "PCAR_PaccarInc_ret_5d", "DHR_vol_20d", "gjr_condvar_h1", "IYR_US_REIT2_zscore_60d", "Nikkei_Japan_zscore_60d", "3M_vol_20d", "AORD_AUS_zscore_60d", "SBUX_vol_20d", "EWL_Switzerland_zscore_60d", "TXN_vol_20d", "TGT_Target_zscore_60d", "ITT_ITTInc_ret_5d", "AMGN_Amgen_ret_1d", "Core_PCE_zscore_60d", "T_ret_1d", "XLB_Materials_zscore_60d"], "is_new": true}, {"model_id": "new_h7_GLOBAL_LightGBM_N30_t4", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 7, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "MO_AltriaMG_ret_1d", "Core_CPI_zscore_60d", "CCI_CrownCastle_vol_20d", "US3Y_Rate_ret_5d", "EWG_Germany_vol_20d", "CI_Cigna_vol_20d", "HD_ret_20d", "INTC_ret_5d", "AMD_ret_5d", "IBEX_Spain_ret_20d", "CTAS_Cintas_vol_20d", "CMCSA_ret_1d", "LUV_SouthwestAir_ret_5d", "SLB_Schlumberger_ret_1d", "DAX_Germany_zscore_60d", "DHR_ret_1d", "TED_Spread_vol_20d", "DIS_vol_20d", "INTC_ret_1d", "XLY_Disc_vol_20d", "LOW_Lowes_ret_20d", "PLD_Prologis_ret_5d", "PAYX_Paychex_vol_20d", "AMD_ret_1d", "ENB_EnbridgeInc_ret_1d", "US3M_Rate_vol_20d", "US1Y_Rate_ret_5d", "ASX_Australia_vol_20d"], "is_new": true}, {"model_id": "new_h7_GLOBAL_LightGBM_N30_t5", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 7, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "AORD_AUS_zscore_60d", "LLY_zscore_60d", "DE_Deere_ret_5d", "SLB_Schlumberger_ret_5d", "LOW_Lowes_ret_5d", "heston_var_ev_h5", "MSTR_Bitcoin3_ret_1d", "DIS_vol_20d", "JNJ_ret_1d", "IWM_SmallCap_vol_20d", "SPY_zscore_60d", "Brent_Oil_FRED_ret_5d", "EWS_Singapore_ret_5d", "NOC_Northrop_ret_20d", "MSTR_Bitcoin3_ret_20d", "NEE_NextEra_ret_20d", "CI_Cigna_vol_20d", "TGT_Target_zscore_60d", "US1Y_Rate_ret_5d", "ASX_Australia_ret_5d", "BA_ret_1d", "PFE_ret_1d", "EFFR_ret_1d", "GILD_Gilead_ret_20d", "MO_AltriaMG_ret_1d", "MRK_Merck_zscore_60d", "heston_ev_h3", "hmm_p_stress"], "is_new": true}, {"model_id": "new_h7_GLOBAL_LightGBM_N30_t6", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 7, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "IYM_BasicMaterials_ret_20d", "MO_AltriaMG_ret_1d", "Michigan_Sentiment_ret_20d", "EWA_Australia_zscore_60d", "CPB_CampbellSoup_zscore_60d", "HD_ret_1d", "EXC_Exelon_zscore_60d", "AORD_AUS_zscore_60d", "GD_GeneralDynamics_zscore_60d", "ITT_ITTInc_ret_5d", "ORCL_vol_20d", "SJM_JM_Smucker_ret_5d", "FedFunds_zscore_60d", "SBUX_vol_20d", "PCAR_PaccarInc_ret_5d", "IWM_SmallCap_vol_20d", "gjr_condvar_h1", "EMR_Emerson_ret_20d", "MSTR_Bitcoin3_ret_20d", "Core_CPI_zscore_60d", "HD_ret_5d", "EOG_EOGResources_vol_20d", "XLB_Materials_zscore_60d", "US6M_Rate_ret_20d", "DIS_vol_20d", "XLV_Health_zscore_60d", "spx_abs_ret_max_5d", "3M_vol_20d"], "is_new": true}, {"model_id": "new_h7_GLOBAL_LightGBM_N30_t7", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 7, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "INTC_ret_1d", "Industrial_Production_zscore_60d", "XOM_ret_20d", "SPY_zscore_60d", "EWY_Korea_ret_20d", "CPB_CampbellSoup_ret_20d", "CCI_CrownCastle_vol_20d", "BDX_Becton_Dickinson_ret_20d", "TGT_Target_zscore_60d", "CPB_CampbellSoup_vol_20d", "ENB_EnbridgeInc_ret_1d", "EWA_Australia_zscore_60d", "GD_GeneralDynamics_zscore_60d", "TM_Telephone_ret_1d", "XLB_Materials_zscore_60d", "spx_abs_ret_max_5d", "ES_Evergy_ret_1d", "EWJ_Japan_vol_20d", "MS_MorganStanley_ret_1d", "HD_zscore_60d", "IYR_US_REIT2_zscore_60d", "CMCSA_ret_1d", "Core_CPI_zscore_60d", "TXN_vol_20d", "Nikkei_Japan_zscore_60d", "IWM_SmallCap_vol_20d", "T10Y2Y_Spread_ret_5d", "CLX_Clorox_vol_20d"], "is_new": true}, {"model_id": "new_h7_GLOBAL_GradientBoosting_N5_t0", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 7, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "QQQ_vol_20d", "WTI_Oil_FRED_zscore_60d", "HD_ret_20d"], "is_new": true}, {"model_id": "new_h7_GLOBAL_GradientBoosting_N5_t1", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 7, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "BLK_BlackRock_zscore_60d", "EWG_Germany_ret_20d", "AMZN_ret_5d"], "is_new": true}, {"model_id": "new_h7_GLOBAL_GradientBoosting_N5_t2", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 7, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "DIS_vol_20d", "gjr_condvar_h1", "EQIX_Equinix_ret_5d"], "is_new": true}, {"model_id": "new_h7_GLOBAL_GradientBoosting_N5_t3", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 7, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "M_Macys_vol_20d", "FedFunds_zscore_60d", "PPL_PPL_ret_1d"], "is_new": true}, {"model_id": "new_h7_GLOBAL_GradientBoosting_N5_t4", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 7, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "Michigan_Sentiment_ret_20d", "AMD_ret_5d", "BTI_BritishAmerican_ret_5d"], "is_new": true}, {"model_id": "new_h7_GLOBAL_GradientBoosting_N5_t5", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 7, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWL_Switzerland_zscore_60d", "AXP_Amex_vol_20d", "Industrial_Production_zscore_60d"], "is_new": true}, {"model_id": "new_h7_GLOBAL_GradientBoosting_N5_t6", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 7, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "WTI_Oil_FRED_zscore_60d", "LOW_Lowes_ret_5d", "XLK_Tech_zscore_60d"], "is_new": true}, {"model_id": "new_h7_GLOBAL_GradientBoosting_N5_t7", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 7, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "MO_AltriaMG_ret_1d", "CCI_CrownCastle_vol_20d", "EWH_HongKong_ret_5d"], "is_new": true}, {"model_id": "new_h7_GLOBAL_GradientBoosting_N8_t0", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 7, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "LMT_LockheedMartin_ret_1d", "HD_ret_5d", "ORCL_vol_20d", "TXN_vol_20d", "EFFR_ret_1d", "EWA_Australia_zscore_60d"], "is_new": true}, {"model_id": "new_h7_GLOBAL_GradientBoosting_N8_t1", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 7, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "Core_PCE_zscore_60d", "DE_Deere_ret_5d", "MO_AltriaMG_ret_1d", "PAYX_Paychex_vol_20d", "EWL_Switzerland_vol_20d", "EWM_Malaysia_ret_1d"], "is_new": true}, {"model_id": "new_h7_GLOBAL_GradientBoosting_N8_t2", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 7, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWY_Korea_ret_20d", "US3M_Rate_vol_20d", "US6M_Rate_ret_20d", "heston_var_ev_h7", "AMD_ret_1d", "spx_vol_5d"], "is_new": true}, {"model_id": "new_h7_GLOBAL_GradientBoosting_N8_t3", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 7, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "ORCL_vol_20d", "M_Macys_vol_20d", "IYM_BasicMaterials_ret_20d", "AMT_AmericanTower_ret_1d", "US3M_Rate_vol_20d", "EWQ_France_ret_20d"], "is_new": true}, {"model_id": "new_h7_GLOBAL_GradientBoosting_N8_t4", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 7, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "CPB_CampbellSoup_ret_5d", "NFCI_ret_5d", "EWC_Canada_zscore_60d", "US1Y_Rate_ret_20d", "TXN_vol_20d", "Brent_Oil_FRED_ret_20d"], "is_new": true}, {"model_id": "new_h7_GLOBAL_GradientBoosting_N8_t5", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 7, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "AMD_ret_5d", "CLX_Clorox_vol_20d", "AMD_ret_1d", "gjr_condvar_h1", "EWA_Australia_ret_1d", "EOG_EOGResources_vol_20d"], "is_new": true}, {"model_id": "new_h7_GLOBAL_GradientBoosting_N8_t6", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 7, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "BTI_BritishAmerican_ret_20d", "3M_vol_20d", "EWA_Australia_ret_1d", "XLK_Tech_zscore_60d", "PG_ret_20d", "HD_zscore_60d"], "is_new": true}, {"model_id": "new_h7_GLOBAL_GradientBoosting_N8_t7", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 7, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "HD_ret_5d", "NWL_Newell_ret_20d", "spx_abs_ret_max_5d", "US3Y_Rate_ret_5d", "heston_ev_h3", "HUM_Humana_ret_5d"], "is_new": true}, {"model_id": "new_h7_GLOBAL_GradientBoosting_N10_t0", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 7, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "PCAR_PaccarInc_ret_5d", "DHR_vol_20d", "AMGN_Amgen_ret_1d", "NVDA_vol_20d", "CPB_CampbellSoup_zscore_60d", "3M_vol_20d", "EQR_Equity_ret_1d", "SLB_Schlumberger_ret_1d"], "is_new": true}, {"model_id": "new_h7_GLOBAL_GradientBoosting_N10_t1", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 7, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "Michigan_Sentiment_ret_20d", "XOM_ret_20d", "spx_momentum_3d", "CPB_CampbellSoup_zscore_60d", "MS_MorganStanley_ret_1d", "GILD_Gilead_ret_20d", "US1Y_Rate_ret_20d", "EWJ_Japan_vol_20d"], "is_new": true}, {"model_id": "new_h7_GLOBAL_GradientBoosting_N10_t2", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 7, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "MS_MorganStanley_zscore_60d", "SLB_Schlumberger_ret_5d", "Brent_Oil_FRED_ret_5d", "EOG_EOGResources_vol_20d", "INTC_ret_5d", "SLB_Schlumberger_ret_1d", "QQQ_vol_20d", "TED_Spread_vol_20d"], "is_new": true}, {"model_id": "new_h7_GLOBAL_GradientBoosting_N10_t3", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 7, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "JNJ_ret_1d", "EQIX_Equinix_ret_5d", "SBUX_zscore_60d", "AVB_AvalonBay_zscore_60d", "Nikkei_Japan_vol_20d", "heston_var_ev_h3", "CPB_CampbellSoup_ret_5d", "DHR_ret_1d"], "is_new": true}, {"model_id": "new_h7_GLOBAL_GradientBoosting_N10_t4", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 7, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "CPB_CampbellSoup_ret_20d", "HD_ret_5d", "EOG_EOGResources_vol_20d", "EWY_Korea_zscore_60d", "INTC_ret_1d", "T_ret_1d", "Nikkei_Japan_zscore_60d", "GE_ret_1d"], "is_new": true}, {"model_id": "new_h7_GLOBAL_GradientBoosting_N10_t5", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 7, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWM_Malaysia_ret_1d", "BTI_BritishAmerican_ret_20d", "EWG_Germany_ret_20d", "TM_Telephone_ret_1d", "HangSeng_HK_ret_5d", "ENB_EnbridgeInc_ret_1d", "US30Y_Rate_ret_20d", "PLD_Prologis_ret_5d"], "is_new": true}, {"model_id": "new_h7_GLOBAL_GradientBoosting_N10_t6", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 7, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "ORCL_zscore_60d", "PAYX_Paychex_vol_20d", "EWG_Germany_vol_20d", "PLD_Prologis_ret_5d", "MO_AltriaMG_ret_1d", "EWJ_Japan_vol_20d", "IBEX_Spain_ret_20d", "SJM_JM_Smucker_ret_1d"], "is_new": true}, {"model_id": "new_h7_GLOBAL_GradientBoosting_N10_t7", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 7, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "HangSeng_HK_ret_1d", "AMD_ret_1d", "DOW_Price_zscore_60d", "EWA_Australia_zscore_60d", "AMD_ret_5d", "LMT_LockheedMartin_vol_20d", "heston_var_ev_h7", "HangSeng_HK_vol_20d"], "is_new": true}, {"model_id": "new_h7_GLOBAL_GradientBoosting_N12_t0", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 7, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EFFR_vol_20d", "HangSeng_HK_vol_20d", "DIS_vol_20d", "T10Y2Y_Spread_ret_5d", "EWJ_Japan_vol_20d", "IYR_US_REIT2_zscore_60d", "heston_var_ev_h3", "VVIX_ret_20d", "XOM_ret_20d", "EXC_Exelon_ret_1d"], "is_new": true}, {"model_id": "new_h7_GLOBAL_GradientBoosting_N12_t1", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 7, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "AMD_ret_5d", "XOM_ret_20d", "Core_PCE_zscore_60d", "HD_ret_1d", "AXP_Amex_vol_20d", "EWQ_France_zscore_60d", "EWM_Malaysia_zscore_60d", "MSTR_Bitcoin3_ret_5d", "SCHW_Schwab_ret_5d", "Nikkei_Japan_vol_20d"], "is_new": true}, {"model_id": "new_h7_GLOBAL_GradientBoosting_N12_t2", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 7, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "LMT_LockheedMartin_ret_1d", "PAYX_Paychex_vol_20d", "US3Y_Rate_ret_5d", "CPB_CampbellSoup_vol_20d", "AORD_AUS_zscore_60d", "AMD_ret_1d", "heston_ev_h3", "EWA_Australia_ret_1d", "EWC_Canada_zscore_60d", "CPB_CampbellSoup_ret_20d"], "is_new": true}, {"model_id": "new_h7_GLOBAL_GradientBoosting_N12_t3", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 7, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "MS_MorganStanley_zscore_60d", "EQIX_Equinix_ret_5d", "ASX_Australia_ret_5d", "SBUX_zscore_60d", "AMD_ret_1d", "US3Y_Rate_ret_5d", "TED_Spread_vol_20d", "MS_MorganStanley_ret_1d", "PAYX_Paychex_zscore_60d", "US5Y_Rate_ret_5d"], "is_new": true}, {"model_id": "new_h7_GLOBAL_GradientBoosting_N12_t4", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 7, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "BTI_BritishAmerican_ret_20d", "M_Macys_vol_20d", "MRK_Merck_zscore_60d", "SJM_JM_Smucker_ret_5d", "LOW_Lowes_ret_5d", "INTC_ret_5d", "SCHW_Schwab_ret_5d", "PAYX_Paychex_ret_20d", "LMT_LockheedMartin_ret_1d", "HangSeng_HK_ret_5d"], "is_new": true}, {"model_id": "new_h7_GLOBAL_GradientBoosting_N12_t5", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 7, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "vix_mean_abs_ret_5d", "ORCL_zscore_60d", "CLX_Clorox_vol_20d", "SBUX_zscore_60d", "ENB_EnbridgeInc_ret_1d", "CCI_CrownCastle_vol_20d", "LMT_LockheedMartin_vol_20d", "NEE_NextEra_ret_20d", "EWA_Australia_zscore_60d", "VOD_Vodafone_zscore_60d"], "is_new": true}, {"model_id": "new_h7_GLOBAL_GradientBoosting_N12_t6", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 7, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "GE_ret_1d", "spx_abs_ret_max_5d", "XOM_ret_1d", "EWG_Germany_ret_20d", "BDX_Becton_Dickinson_ret_20d", "HUM_Humana_ret_5d", "BA_ret_1d", "vix_mean_abs_ret_5d", "Nikkei_Japan_vol_20d", "HD_ret_5d"], "is_new": true}, {"model_id": "new_h7_GLOBAL_GradientBoosting_N12_t7", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 7, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "DHR_vol_20d", "PPL_PPL_ret_1d", "3M_ret_5d", "GD_GeneralDynamics_zscore_60d", "XLK_Tech_zscore_60d", "EWH_HongKong_ret_5d", "vix_acceleration_1d", "US3Y_Rate_ret_5d", "MSTR_Bitcoin3_ret_20d", "GE_ret_1d"], "is_new": true}, {"model_id": "new_h7_GLOBAL_GradientBoosting_N15_t0", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 7, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "hmm_p_stress", "EXC_Exelon_zscore_60d", "HD_zscore_60d", "XLK_Tech_zscore_60d", "HD_ret_1d", "ASX_Australia_vol_20d", "DAX_Germany_vol_20d", "EWQ_France_zscore_60d", "MSTR_Bitcoin3_ret_20d", "CI_Cigna_vol_20d", "NFCI_ret_5d", "AVB_AvalonBay_zscore_60d", "PFE_ret_1d"], "is_new": true}, {"model_id": "new_h7_GLOBAL_GradientBoosting_N15_t1", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 7, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "BDX_Becton_Dickinson_ret_20d", "AMT_AmericanTower_ret_1d", "BTI_BritishAmerican_ret_20d", "DHR_vol_20d", "TM_Telephone_ret_1d", "EWJ_Japan_vol_20d", "hmm_p_stress", "ORCL_zscore_60d", "ASX_Australia_vol_20d", "DOW_Price_zscore_60d", "LMT_LockheedMartin_ret_1d", "NEE_NextEra_ret_20d", "EOG_EOGResources_vol_20d"], "is_new": true}, {"model_id": "new_h7_GLOBAL_GradientBoosting_N15_t2", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 7, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "Michigan_Sentiment_ret_20d", "DAX_Germany_zscore_60d", "heston_ev_h3", "BA_ret_1d", "HangSeng_HK_ret_1d", "EWL_Switzerland_zscore_60d", "SLB_Schlumberger_ret_1d", "US6M_Rate_ret_20d", "heston_var_ev_h3", "LLY_zscore_60d", "BDX_Becton_Dickinson_ret_20d", "IYR_US_REIT2_zscore_60d", "GILD_Gilead_ret_20d"], "is_new": true}, {"model_id": "new_h7_GLOBAL_GradientBoosting_N15_t3", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 7, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "INTC_ret_5d", "TED_Spread_vol_20d", "PG_ret_20d", "Core_PCE_zscore_60d", "DOW_Price_zscore_60d", "AXP_Amex_ret_20d", "LMT_LockheedMartin_vol_20d", "BA_ret_1d", "CTAS_Cintas_vol_20d", "DE_Deere_ret_5d", "DIS_vol_20d", "spx_vol_5d", "EWG_Germany_vol_20d"], "is_new": true}, {"model_id": "new_h7_GLOBAL_GradientBoosting_N15_t4", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 7, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "DOW_Price_zscore_60d", "AMT_AmericanTower_ret_1d", "WTI_Oil_FRED_zscore_60d", "BLK_BlackRock_zscore_60d", "IBEX_Spain_ret_20d", "MSTR_Bitcoin3_ret_5d", "Core_PCE_zscore_60d", "DHR_ret_1d", "XLY_Disc_vol_20d", "M_Macys_vol_20d", "EWS_Singapore_ret_5d", "Brent_Oil_FRED_ret_5d", "Brent_Oil_FRED_ret_20d"], "is_new": true}, {"model_id": "new_h7_GLOBAL_GradientBoosting_N15_t5", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 7, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "PPL_PPL_ret_1d", "Brent_Oil_FRED_ret_20d", "NOC_Northrop_ret_20d", "DE_Deere_ret_5d", "SJM_JM_Smucker_ret_1d", "US1Y_Rate_ret_5d", "CCI_CrownCastle_vol_20d", "GE_ret_1d", "LMT_LockheedMartin_vol_20d", "hmm_p_stress", "MSTR_Bitcoin3_ret_5d", "PLD_Prologis_ret_5d", "HD_ret_20d"], "is_new": true}, {"model_id": "new_h7_GLOBAL_GradientBoosting_N15_t6", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 7, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EOG_EOGResources_ret_5d", "Michigan_Sentiment_ret_20d", "DAX_Germany_vol_20d", "spx_momentum_3d", "LOW_Lowes_ret_20d", "T10Y2Y_Spread_ret_5d", "NOC_Northrop_ret_20d", "CPB_CampbellSoup_zscore_60d", "US1Y_Rate_ret_20d", "XOM_ret_20d", "MSTR_Bitcoin3_ret_1d", "Brent_Oil_FRED_ret_20d", "EWQ_France_zscore_60d"], "is_new": true}, {"model_id": "new_h7_GLOBAL_GradientBoosting_N15_t7", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 7, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "M_Macys_vol_20d", "XOM_ret_20d", "CMCSA_ret_1d", "spx_momentum_3d", "EWA_Australia_zscore_60d", "PAYX_Paychex_zscore_60d", "US5Y_Rate_ret_5d", "T_ret_1d", "TM_Telephone_vol_20d", "NOC_Northrop_ret_20d", "PG_ret_20d", "HUM_Humana_ret_5d", "EWY_Korea_zscore_60d"], "is_new": true}, {"model_id": "new_h7_GLOBAL_GradientBoosting_N20_t0", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 7, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "PAYX_Paychex_ret_20d", "US1Y_Rate_ret_5d", "Nikkei_Japan_vol_20d", "spx_momentum_3d", "EOG_EOGResources_ret_5d", "Michigan_Sentiment_ret_20d", "LMT_LockheedMartin_vol_20d", "VRP_ma5", "Core_PCE_zscore_60d", "DAX_Germany_vol_20d", "Retail_Sales_zscore_60d", "XLB_Materials_zscore_60d", "XLY_Disc_vol_20d", "MS_MorganStanley_ret_1d", "EWA_Australia_zscore_60d", "US3M_Rate_zscore_60d", "CI_Cigna_vol_20d", "BTI_BritishAmerican_ret_5d"], "is_new": true}, {"model_id": "new_h7_GLOBAL_GradientBoosting_N20_t1", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 7, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "GILD_Gilead_ret_20d", "AMGN_Amgen_ret_1d", "vix_mean_abs_ret_5d", "EQR_Equity_ret_1d", "spx_momentum_3d", "XOM_ret_20d", "MRK_Merck_zscore_60d", "SLB_Schlumberger_ret_1d", "MSTR_Bitcoin3_ret_5d", "EMR_Emerson_ret_20d", "NEE_NextEra_ret_20d", "EWJ_Japan_vol_20d", "TGT_Target_zscore_60d", "AORD_AUS_zscore_60d", "SPY_zscore_60d", "MSTR_Bitcoin3_ret_20d", "XLF_Fin_vol_20d", "ENB_EnbridgeInc_ret_1d"], "is_new": true}, {"model_id": "new_h7_GLOBAL_GradientBoosting_N20_t2", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 7, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "AMD_ret_5d", "TED_Spread_vol_20d", "PCAR_PaccarInc_ret_5d", "EWQ_France_ret_20d", "QQQ_vol_20d", "PAYX_Paychex_zscore_60d", "HangSeng_HK_ret_1d", "EOG_EOGResources_vol_20d", "heston_var_ev_h5", "HUM_Humana_ret_5d", "IBEX_Spain_ret_20d", "EWS_Singapore_ret_5d", "INTC_ret_5d", "Nikkei_Japan_vol_20d", "SO_SouthernCo_ret_5d", "3M_ret_5d", "SBUX_ret_5d", "SJM_JM_Smucker_ret_5d"], "is_new": true}, {"model_id": "new_h7_GLOBAL_GradientBoosting_N20_t3", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 7, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "DE_Deere_ret_5d", "ORCL_zscore_60d", "US5Y_Rate_ret_5d", "EWL_Switzerland_zscore_60d", "AXP_Amex_ret_20d", "Industrial_Production_zscore_60d", "PLD_Prologis_ret_5d", "VVIX_ret_20d", "MS_MorganStanley_ret_1d", "IBEX_Spain_ret_20d", "EWM_Malaysia_zscore_60d", "T_ret_1d", "US1Y_Rate_ret_20d", "PAYX_Paychex_vol_20d", "EWA_Australia_zscore_60d", "EWH_HongKong_ret_5d", "HD_ret_20d", "MSTR_Bitcoin3_ret_1d"], "is_new": true}, {"model_id": "new_h7_GLOBAL_GradientBoosting_N20_t4", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 7, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "HD_ret_1d", "GE_ret_1d", "DAX_Germany_zscore_60d", "TM_Telephone_ret_1d", "PCAR_PaccarInc_ret_5d", "ENB_EnbridgeInc_ret_1d", "EOG_EOGResources_ret_5d", "EWL_Switzerland_zscore_60d", "CPB_CampbellSoup_ret_5d", "XOM_ret_20d", "EWA_Australia_ret_1d", "3M_vol_20d", "GILD_Gilead_ret_20d", "3M_ret_5d", "heston_var_ev_h3", "IYR_US_REIT2_zscore_60d", "BTI_BritishAmerican_ret_20d", "EMR_Emerson_ret_20d"], "is_new": true}, {"model_id": "new_h7_GLOBAL_GradientBoosting_N20_t5", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 7, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "IYM_BasicMaterials_ret_20d", "AMT_AmericanTower_ret_1d", "HangSeng_HK_ret_1d", "XLF_Fin_vol_20d", "LOW_Lowes_ret_20d", "EWC_Canada_zscore_60d", "ORCL_vol_20d", "hmm_p_stress", "HD_ret_5d", "AXP_Amex_vol_20d", "EWQ_France_ret_20d", "LMT_LockheedMartin_vol_20d", "EFFR_vol_20d", "ORCL_zscore_60d", "ASX_Australia_vol_20d", "EWJ_Japan_vol_20d", "gjr_condvar_h1", "EOG_EOGResources_ret_5d"], "is_new": true}, {"model_id": "new_h7_GLOBAL_GradientBoosting_N20_t6", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 7, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWM_Malaysia_ret_1d", "MSTR_Bitcoin3_ret_20d", "DHR_ret_1d", "M_Macys_vol_20d", "EWL_Switzerland_zscore_60d", "US5Y_Rate_ret_5d", "TM_Telephone_vol_20d", "EWQ_France_ret_20d", "SJM_JM_Smucker_ret_5d", "GILD_Gilead_ret_20d", "ES_Evergy_ret_1d", "HUM_Humana_ret_5d", "TXN_vol_20d", "CI_Cigna_vol_20d", "DOW_Price_zscore_60d", "BLK_BlackRock_zscore_60d", "3M_ret_5d", "US30Y_Rate_ret_20d"], "is_new": true}, {"model_id": "new_h7_GLOBAL_GradientBoosting_N20_t7", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 7, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWM_Malaysia_ret_1d", "MRK_Merck_zscore_60d", "DHR_vol_20d", "NFCI_ret_5d", "DAX_Germany_zscore_60d", "EMR_Emerson_ret_20d", "SLB_Schlumberger_ret_5d", "US5Y_Rate_ret_5d", "CMCSA_ret_1d", "AMGN_Amgen_ret_1d", "T_ret_1d", "LMT_LockheedMartin_vol_20d", "AVB_AvalonBay_zscore_60d", "ASX_Australia_ret_5d", "WTI_Oil_FRED_zscore_60d", "CPB_CampbellSoup_ret_5d", "SBUX_vol_20d", "MS_MorganStanley_ret_5d"], "is_new": true}, {"model_id": "new_h7_GLOBAL_GradientBoosting_N25_t0", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 7, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "BA_ret_1d", "LUV_SouthwestAir_ret_5d", "ASX_Australia_vol_20d", "EWA_Australia_ret_1d", "CPB_CampbellSoup_ret_5d", "VOD_Vodafone_zscore_60d", "SCHW_Schwab_ret_5d", "CI_Cigna_vol_20d", "CCI_CrownCastle_vol_20d", "Core_CPI_zscore_60d", "PCAR_PaccarInc_ret_5d", "3M_vol_20d", "TM_Telephone_ret_1d", "SBUX_vol_20d", "US30Y_Rate_ret_20d", "DHR_vol_20d", "EFFR_ret_1d", "EWM_Malaysia_zscore_60d", "SO_SouthernCo_ret_5d", "IYR_US_REIT2_zscore_60d", "VVIX_ret_20d", "HUM_Humana_ret_5d", "heston_var_ev_h7"], "is_new": true}, {"model_id": "new_h7_GLOBAL_GradientBoosting_N25_t1", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 7, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "SBUX_ret_5d", "INTC_ret_1d", "ASX_Australia_ret_5d", "heston_var_ev_h3", "MS_MorganStanley_ret_1d", "EWC_Canada_zscore_60d", "HD_ret_1d", "EWY_Korea_zscore_60d", "PLD_Prologis_ret_5d", "AMD_ret_1d", "US7Y_Rate_ret_20d", "GE_ret_1d", "HD_ret_5d", "HangSeng_HK_ret_1d", "EQIX_Equinix_ret_5d", "SCHW_Schwab_ret_5d", "FedFunds_zscore_60d", "BA_ret_1d", "Retail_Sales_zscore_60d", "Michigan_Sentiment_ret_20d", "NVDA_vol_20d", "heston_var_ev_h7", "XLY_Disc_vol_20d"], "is_new": true}, {"model_id": "new_h7_GLOBAL_GradientBoosting_N25_t2", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 7, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "DAX_Germany_vol_20d", "US3Y_Rate_ret_5d", "ENB_EnbridgeInc_ret_1d", "AXP_Amex_ret_20d", "SO_SouthernCo_ret_5d", "heston_ev_h3", "IBEX_Spain_ret_20d", "BTI_BritishAmerican_ret_5d", "DHR_ret_1d", "DHR_vol_20d", "DIS_vol_20d", "ASX_Australia_vol_20d", "EWQ_France_ret_20d", "TM_Telephone_vol_20d", "SCHW_Schwab_ret_5d", "Nikkei_Japan_zscore_60d", "PAYX_Paychex_zscore_60d", "EWH_HongKong_ret_5d", "TGT_Target_zscore_60d", "HangSeng_HK_ret_5d", "heston_var_ev_h3", "US7Y_Rate_ret_20d", "BTI_BritishAmerican_ret_20d"], "is_new": true}, {"model_id": "new_h7_GLOBAL_GradientBoosting_N25_t3", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 7, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "DHR_vol_20d", "DHR_ret_1d", "EWS_Singapore_ret_5d", "US6M_Rate_ret_20d", "CPB_CampbellSoup_zscore_60d", "SJM_JM_Smucker_ret_5d", "DE_Deere_vol_20d", "HangSeng_HK_ret_1d", "LUV_SouthwestAir_ret_5d", "CPB_CampbellSoup_ret_20d", "HD_ret_5d", "ASX_Australia_ret_5d", "NFCI_ret_5d", "MO_AltriaMG_ret_1d", "EWQ_France_ret_20d", "HUM_Humana_ret_5d", "AVB_AvalonBay_zscore_60d", "ENB_EnbridgeInc_ret_1d", "EQR_Equity_ret_1d", "MS_MorganStanley_ret_5d", "MSTR_Bitcoin3_ret_5d", "SJM_JM_Smucker_ret_1d", "EWL_Switzerland_vol_20d"], "is_new": true}, {"model_id": "new_h7_GLOBAL_GradientBoosting_N25_t4", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 7, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "AORD_AUS_zscore_60d", "HangSeng_HK_vol_20d", "EWA_Australia_ret_1d", "SCHW_Schwab_ret_5d", "FedFunds_zscore_60d", "HD_ret_5d", "EWQ_France_zscore_60d", "BTI_BritishAmerican_ret_5d", "WTI_Oil_FRED_zscore_60d", "PAYX_Paychex_vol_20d", "LUV_SouthwestAir_ret_5d", "EXC_Exelon_zscore_60d", "DE_Deere_vol_20d", "EWM_Malaysia_ret_1d", "US6M_Rate_ret_20d", "PCAR_PaccarInc_ret_5d", "EFFR_ret_1d", "SO_SouthernCo_ret_5d", "AMD_ret_1d", "CPB_CampbellSoup_vol_20d", "US30Y_Rate_ret_20d", "spx_momentum_3d", "AMT_AmericanTower_ret_1d"], "is_new": true}, {"model_id": "new_h7_GLOBAL_GradientBoosting_N25_t5", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 7, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWM_Malaysia_zscore_60d", "spx_momentum_3d", "CMCSA_ret_1d", "Core_CPI_zscore_60d", "LOW_Lowes_ret_20d", "TED_Spread_zscore_60d", "PCAR_PaccarInc_ret_5d", "AVB_AvalonBay_zscore_60d", "heston_var_ev_h7", "WTI_Oil_FRED_zscore_60d", "CCI_CrownCastle_vol_20d", "FedFunds_zscore_60d", "US1Y_Rate_ret_20d", "CPB_CampbellSoup_zscore_60d", "EWA_Australia_ret_1d", "XLB_Materials_zscore_60d", "SPY_zscore_60d", "EMR_Emerson_ret_20d", "US6M_Rate_ret_20d", "SO_SouthernCo_ret_5d", "Brent_Oil_FRED_ret_5d", "3M_vol_20d", "VVIX_ret_20d"], "is_new": true}, {"model_id": "new_h7_GLOBAL_GradientBoosting_N25_t6", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 7, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "gjr_condvar_h1", "CPB_CampbellSoup_ret_5d", "LLY_zscore_60d", "VOD_Vodafone_zscore_60d", "EWM_Malaysia_ret_1d", "TED_Spread_zscore_60d", "IYM_BasicMaterials_ret_20d", "BA_ret_1d", "VRP_ma5", "TM_Telephone_ret_1d", "Brent_Oil_FRED_ret_20d", "T10Y2Y_Spread_ret_5d", "BTI_BritishAmerican_ret_5d", "HD_ret_1d", "US7Y_Rate_ret_20d", "MSTR_Bitcoin3_ret_1d", "US3M_Rate_zscore_60d", "NOC_Northrop_ret_20d", "GE_ret_1d", "LUV_SouthwestAir_ret_5d", "EWJ_Japan_vol_20d", "EWQ_France_zscore_60d", "PCAR_PaccarInc_ret_5d"], "is_new": true}, {"model_id": "new_h7_GLOBAL_GradientBoosting_N25_t7", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 7, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWM_Malaysia_vol_20d", "vix_mean_abs_ret_5d", "EWA_Australia_ret_1d", "HangSeng_HK_ret_1d", "EWH_HongKong_ret_5d", "GE_ret_1d", "Brent_Oil_FRED_ret_20d", "DAX_Germany_vol_20d", "XOM_ret_20d", "SLB_Schlumberger_ret_1d", "EQIX_Equinix_ret_5d", "LOW_Lowes_ret_5d", "CPB_CampbellSoup_vol_20d", "HD_ret_1d", "TGT_Target_zscore_60d", "M_Macys_vol_20d", "EWY_Korea_zscore_60d", "AORD_AUS_zscore_60d", "DHR_vol_20d", "ENB_EnbridgeInc_ret_1d", "VOD_Vodafone_zscore_60d", "FedFunds_zscore_60d", "SJM_JM_Smucker_ret_5d"], "is_new": true}, {"model_id": "new_h7_GLOBAL_GradientBoosting_N30_t0", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 7, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "AMD_ret_1d", "HangSeng_HK_ret_5d", "SPY_zscore_60d", "ORCL_zscore_60d", "EOG_EOGResources_ret_5d", "TED_Spread_vol_20d", "PAYX_Paychex_vol_20d", "HangSeng_HK_vol_20d", "EWM_Malaysia_ret_1d", "XLV_Health_zscore_60d", "HD_ret_20d", "EWA_Australia_ret_1d", "LMT_LockheedMartin_ret_1d", "US1Y_Rate_ret_5d", "MS_MorganStanley_ret_1d", "DE_Deere_vol_20d", "Core_PCE_zscore_60d", "EWQ_France_zscore_60d", "DAX_Germany_vol_20d", "TGT_Target_zscore_60d", "GE_ret_1d", "CCI_CrownCastle_vol_20d", "US30Y_Rate_ret_20d", "AVB_AvalonBay_zscore_60d", "Retail_Sales_zscore_60d", "Core_CPI_zscore_60d", "PAYX_Paychex_ret_20d", "US7Y_Rate_ret_20d"], "is_new": true}, {"model_id": "new_h7_GLOBAL_GradientBoosting_N30_t1", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 7, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "HangSeng_HK_ret_5d", "Brent_Oil_FRED_ret_20d", "SCHW_Schwab_ret_5d", "EWG_Germany_vol_20d", "EQIX_Equinix_ret_5d", "EWA_Australia_zscore_60d", "HUM_Humana_ret_5d", "EFFR_vol_20d", "IYR_US_REIT2_zscore_60d", "DAX_Germany_vol_20d", "EWC_Canada_zscore_60d", "SLB_Schlumberger_ret_1d", "ORCL_vol_20d", "Core_PCE_zscore_60d", "ORCL_zscore_60d", "INTC_ret_5d", "XLK_Tech_zscore_60d", "NVDA_vol_20d", "IWM_SmallCap_vol_20d", "BTI_BritishAmerican_ret_20d", "PFE_ret_1d", "spx_momentum_3d", "EXC_Exelon_zscore_60d", "EOG_EOGResources_ret_5d", "M_Macys_vol_20d", "DOW_Price_zscore_60d", "ASX_Australia_ret_5d", "NWL_Newell_ret_20d"], "is_new": true}, {"model_id": "new_h7_GLOBAL_GradientBoosting_N30_t2", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 7, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "SPY_zscore_60d", "MRK_Merck_zscore_60d", "ENB_EnbridgeInc_ret_1d", "AMZN_ret_5d", "LOW_Lowes_ret_20d", "TED_Spread_vol_20d", "FedFunds_zscore_60d", "PCAR_PaccarInc_ret_5d", "heston_var_ev_h3", "3M_ret_5d", "TM_Telephone_vol_20d", "JNJ_ret_1d", "HD_ret_5d", "Brent_Oil_FRED_ret_20d", "DE_Deere_ret_5d", "XOM_ret_1d", "GE_ret_1d", "SBUX_vol_20d", "T10Y2Y_Spread_ret_5d", "US1Y_Rate_ret_5d", "XLK_Tech_zscore_60d", "EWJ_Japan_vol_20d", "BA_ret_1d", "AMGN_Amgen_ret_1d", "Industrial_Production_zscore_60d", "US6M_Rate_ret_20d", "PG_ret_20d", "EWA_Australia_zscore_60d"], "is_new": true}, {"model_id": "new_h7_GLOBAL_GradientBoosting_N30_t3", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 7, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EQIX_Equinix_ret_5d", "EXC_Exelon_ret_1d", "IBEX_Spain_ret_20d", "EWY_Korea_zscore_60d", "XLK_Tech_zscore_60d", "DOW_Price_zscore_60d", "US7Y_Rate_ret_20d", "JNJ_ret_1d", "PPL_PPL_ret_1d", "VVIX_ret_20d", "PFE_ret_1d", "T_ret_1d", "MSTR_Bitcoin3_ret_5d", "Core_CPI_zscore_60d", "DE_Deere_vol_20d", "US30Y_Rate_ret_20d", "Michigan_Sentiment_ret_20d", "3M_vol_20d", "BDX_Becton_Dickinson_ret_20d", "EWJ_Japan_vol_20d", "vix_acceleration_1d", "DHR_ret_1d", "ENB_EnbridgeInc_ret_1d", "heston_var_ev_h7", "MO_AltriaMG_ret_1d", "XOM_ret_1d", "DE_Deere_ret_5d", "GILD_Gilead_ret_20d"], "is_new": true}, {"model_id": "new_h7_GLOBAL_GradientBoosting_N30_t4", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 7, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "Brent_Oil_FRED_ret_5d", "MS_MorganStanley_zscore_60d", "MS_MorganStanley_ret_5d", "NEE_NextEra_ret_20d", "EWM_Malaysia_vol_20d", "3M_ret_5d", "EWS_Singapore_ret_5d", "vix_mean_abs_ret_5d", "SCHW_Schwab_ret_5d", "DOW_Price_zscore_60d", "Core_CPI_zscore_60d", "CPB_CampbellSoup_ret_5d", "NWL_Newell_ret_20d", "EMR_Emerson_ret_20d", "HangSeng_HK_ret_5d", "HD_ret_1d", "EOG_EOGResources_ret_5d", "BTI_BritishAmerican_ret_5d", "NOC_Northrop_ret_20d", "US3Y_Rate_ret_5d", "MSTR_Bitcoin3_ret_20d", "XLB_Materials_zscore_60d", "PAYX_Paychex_ret_20d", "Nikkei_Japan_zscore_60d", "AVB_AvalonBay_zscore_60d", "hmm_p_stress", "3M_vol_20d", "HangSeng_HK_ret_1d"], "is_new": true}, {"model_id": "new_h7_GLOBAL_GradientBoosting_N30_t5", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 7, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "MS_MorganStanley_zscore_60d", "NEE_NextEra_ret_20d", "MSTR_Bitcoin3_ret_5d", "vix_mean_abs_ret_5d", "DE_Deere_vol_20d", "MS_MorganStanley_ret_5d", "ASX_Australia_ret_5d", "SLB_Schlumberger_ret_1d", "XLV_Health_zscore_60d", "Industrial_Production_zscore_60d", "IYM_BasicMaterials_ret_20d", "T_ret_1d", "heston_var_ev_h5", "EWC_Canada_zscore_60d", "QQQ_vol_20d", "Core_PCE_zscore_60d", "AXP_Amex_ret_20d", "EWH_HongKong_ret_5d", "gjr_condvar_h1", "WTI_Oil_FRED_zscore_60d", "US30Y_Rate_ret_20d", "GE_ret_1d", "LOW_Lowes_ret_20d", "XLB_Materials_zscore_60d", "SJM_JM_Smucker_ret_1d", "US5Y_Rate_ret_5d", "US3M_Rate_zscore_60d", "Retail_Sales_zscore_60d"], "is_new": true}, {"model_id": "new_h7_GLOBAL_GradientBoosting_N30_t6", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 7, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "IBEX_Spain_ret_20d", "AMD_ret_1d", "EWY_Korea_zscore_60d", "SJM_JM_Smucker_ret_1d", "CI_Cigna_vol_20d", "LMT_LockheedMartin_ret_1d", "VVIX_ret_20d", "DHR_ret_1d", "US3M_Rate_zscore_60d", "CTAS_Cintas_vol_20d", "heston_var_ev_h5", "Core_PCE_zscore_60d", "EWY_Korea_ret_20d", "LLY_zscore_60d", "SJM_JM_Smucker_ret_5d", "TGT_Target_zscore_60d", "SCHW_Schwab_ret_5d", "AORD_AUS_zscore_60d", "spx_vol_5d", "EWG_Germany_vol_20d", "DHR_vol_20d", "ASX_Australia_vol_20d", "TM_Telephone_vol_20d", "AMGN_Amgen_ret_1d", "FedFunds_zscore_60d", "NVDA_vol_20d", "SPY_zscore_60d", "BLK_BlackRock_zscore_60d"], "is_new": true}, {"model_id": "new_h7_GLOBAL_GradientBoosting_N30_t7", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 7, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "Retail_Sales_zscore_60d", "EWC_Canada_zscore_60d", "vix_acceleration_1d", "GE_ret_1d", "hmm_p_stress", "MO_AltriaMG_ret_1d", "VOD_Vodafone_zscore_60d", "SJM_JM_Smucker_ret_5d", "EWY_Korea_zscore_60d", "AVB_AvalonBay_zscore_60d", "SBUX_ret_5d", "CPB_CampbellSoup_zscore_60d", "PG_ret_20d", "IWM_SmallCap_vol_20d", "EOG_EOGResources_ret_5d", "HUM_Humana_ret_5d", "PAYX_Paychex_vol_20d", "Nikkei_Japan_vol_20d", "VVIX_ret_20d", "EWM_Malaysia_zscore_60d", "US1Y_Rate_ret_20d", "IYM_BasicMaterials_ret_20d", "AMGN_Amgen_ret_1d", "DOW_Price_zscore_60d", "3M_vol_20d", "AMD_ret_1d", "ITT_ITTInc_ret_5d", "ES_Evergy_ret_1d"], "is_new": true}, {"model_id": "new_h7_GLOBAL_RandomForest_N5_t0", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 7, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EFFR_vol_20d", "MS_MorganStanley_ret_5d", "hmm_p_stress"], "is_new": true}, {"model_id": "new_h7_GLOBAL_RandomForest_N5_t1", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 7, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "Core_CPI_zscore_60d", "EWA_Australia_ret_1d", "HD_ret_1d"], "is_new": true}, {"model_id": "new_h7_GLOBAL_RandomForest_N5_t2", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 7, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "HUM_Humana_ret_5d", "3M_vol_20d", "3M_ret_5d"], "is_new": true}, {"model_id": "new_h7_GLOBAL_RandomForest_N5_t3", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 7, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "HD_zscore_60d", "EWQ_France_zscore_60d", "HangSeng_HK_ret_5d"], "is_new": true}, {"model_id": "new_h7_GLOBAL_RandomForest_N5_t4", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 7, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWA_Australia_ret_1d", "US1Y_Rate_ret_20d", "SPY_zscore_60d"], "is_new": true}, {"model_id": "new_h7_GLOBAL_RandomForest_N5_t5", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 7, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EFFR_ret_1d", "EFFR_vol_20d", "HangSeng_HK_ret_1d"], "is_new": true}, {"model_id": "new_h7_GLOBAL_RandomForest_N5_t6", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 7, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "US3M_Rate_vol_20d", "INTC_ret_1d", "HangSeng_HK_vol_20d"], "is_new": true}, {"model_id": "new_h7_GLOBAL_RandomForest_N5_t7", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 7, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "CPB_CampbellSoup_ret_20d", "AMT_AmericanTower_ret_1d", "CLX_Clorox_vol_20d"], "is_new": true}, {"model_id": "new_h7_GLOBAL_RandomForest_N8_t0", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 7, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "T_ret_1d", "EWL_Switzerland_zscore_60d", "QQQ_vol_20d", "AMGN_Amgen_ret_1d", "ORCL_zscore_60d", "XLV_Health_zscore_60d"], "is_new": true}, {"model_id": "new_h7_GLOBAL_RandomForest_N8_t1", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 7, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EFFR_vol_20d", "EQIX_Equinix_ret_5d", "HD_ret_1d", "gjr_condvar_h1", "ENB_EnbridgeInc_ret_1d", "PG_ret_20d"], "is_new": true}, {"model_id": "new_h7_GLOBAL_RandomForest_N8_t2", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 7, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "MS_MorganStanley_ret_5d", "EWQ_France_ret_20d", "DHR_vol_20d", "HD_ret_1d", "CI_Cigna_vol_20d", "SJM_JM_Smucker_ret_5d"], "is_new": true}, {"model_id": "new_h7_GLOBAL_RandomForest_N8_t3", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 7, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "US30Y_Rate_ret_20d", "IBEX_Spain_ret_20d", "BTI_BritishAmerican_ret_20d", "LOW_Lowes_ret_20d", "DHR_ret_1d", "EWQ_France_ret_20d"], "is_new": true}, {"model_id": "new_h7_GLOBAL_RandomForest_N8_t4", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 7, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "Brent_Oil_FRED_ret_20d", "DE_Deere_ret_5d", "EXC_Exelon_ret_1d", "EWG_Germany_vol_20d", "TED_Spread_zscore_60d", "PPL_PPL_ret_1d"], "is_new": true}, {"model_id": "new_h7_GLOBAL_RandomForest_N8_t5", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 7, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "XLF_Fin_vol_20d", "EWG_Germany_vol_20d", "CTAS_Cintas_vol_20d", "HD_ret_5d", "AORD_AUS_zscore_60d", "DOW_Price_zscore_60d"], "is_new": true}, {"model_id": "new_h7_GLOBAL_RandomForest_N8_t6", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 7, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "VRP_ma5", "EWA_Australia_ret_1d", "EWJ_Japan_vol_20d", "SBUX_ret_5d", "EOG_EOGResources_ret_5d", "US7Y_Rate_ret_20d"], "is_new": true}, {"model_id": "new_h7_GLOBAL_RandomForest_N8_t7", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 7, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "LMT_LockheedMartin_ret_1d", "LLY_zscore_60d", "AMGN_Amgen_ret_1d", "hmm_p_stress", "Nikkei_Japan_zscore_60d", "DIS_vol_20d"], "is_new": true}, {"model_id": "new_h7_GLOBAL_RandomForest_N10_t0", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 7, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "GD_GeneralDynamics_zscore_60d", "XOM_ret_20d", "SBUX_zscore_60d", "NFCI_ret_5d", "EWH_HongKong_ret_5d", "EWM_Malaysia_zscore_60d", "AMD_ret_1d", "CLX_Clorox_vol_20d"], "is_new": true}, {"model_id": "new_h7_GLOBAL_RandomForest_N10_t1", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 7, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "DAX_Germany_vol_20d", "HD_ret_1d", "FedFunds_zscore_60d", "vix_mean_abs_ret_5d", "heston_var_ev_h3", "US5Y_Rate_ret_5d", "T10Y2Y_Spread_ret_5d", "DE_Deere_ret_5d"], "is_new": true}, {"model_id": "new_h7_GLOBAL_RandomForest_N10_t2", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 7, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "Brent_Oil_FRED_ret_20d", "AORD_AUS_zscore_60d", "Core_PCE_zscore_60d", "DAX_Germany_zscore_60d", "PAYX_Paychex_zscore_60d", "EWL_Switzerland_vol_20d", "US3Y_Rate_ret_5d", "Retail_Sales_zscore_60d"], "is_new": true}, {"model_id": "new_h7_GLOBAL_RandomForest_N10_t3", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 7, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "Core_PCE_zscore_60d", "M_Macys_vol_20d", "SBUX_vol_20d", "T_ret_1d", "T10Y2Y_Spread_ret_5d", "MO_AltriaMG_ret_1d", "AMGN_Amgen_ret_1d", "Core_CPI_zscore_60d"], "is_new": true}, {"model_id": "new_h7_GLOBAL_RandomForest_N10_t4", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 7, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "CI_Cigna_vol_20d", "EWL_Switzerland_zscore_60d", "heston_ev_h3", "CCI_CrownCastle_vol_20d", "MRK_Merck_zscore_60d", "AORD_AUS_zscore_60d", "vix_acceleration_1d", "TM_Telephone_vol_20d"], "is_new": true}, {"model_id": "new_h7_GLOBAL_RandomForest_N10_t5", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 7, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "PAYX_Paychex_zscore_60d", "EOG_EOGResources_ret_5d", "US3Y_Rate_ret_5d", "QQQ_vol_20d", "HangSeng_HK_ret_1d", "Brent_Oil_FRED_ret_20d", "EMR_Emerson_ret_20d", "SBUX_ret_5d"], "is_new": true}, {"model_id": "new_h7_GLOBAL_RandomForest_N10_t6", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 7, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWA_Australia_zscore_60d", "HD_ret_5d", "EQR_Equity_ret_1d", "EWM_Malaysia_vol_20d", "ENB_EnbridgeInc_ret_1d", "DHR_vol_20d", "EWG_Germany_vol_20d", "ASX_Australia_vol_20d"], "is_new": true}, {"model_id": "new_h7_GLOBAL_RandomForest_N10_t7", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 7, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EQR_Equity_ret_1d", "MSTR_Bitcoin3_ret_5d", "T_ret_1d", "DE_Deere_vol_20d", "CPB_CampbellSoup_vol_20d", "EWQ_France_ret_20d", "EWG_Germany_ret_20d", "GD_GeneralDynamics_zscore_60d"], "is_new": true}, {"model_id": "new_h7_GLOBAL_RandomForest_N12_t0", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 7, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWM_Malaysia_ret_1d", "NOC_Northrop_ret_20d", "DIS_vol_20d", "DE_Deere_vol_20d", "EWC_Canada_zscore_60d", "VOD_Vodafone_zscore_60d", "TED_Spread_vol_20d", "ES_Evergy_ret_1d", "SBUX_ret_5d", "BDX_Becton_Dickinson_ret_20d"], "is_new": true}, {"model_id": "new_h7_GLOBAL_RandomForest_N12_t1", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 7, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "XLV_Health_zscore_60d", "GILD_Gilead_ret_20d", "EWA_Australia_ret_1d", "US3Y_Rate_ret_5d", "SBUX_ret_5d", "heston_ev_h3", "EOG_EOGResources_ret_5d", "FedFunds_zscore_60d", "DE_Deere_vol_20d", "EQR_Equity_ret_1d"], "is_new": true}, {"model_id": "new_h7_GLOBAL_RandomForest_N12_t2", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 7, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EFFR_vol_20d", "spx_momentum_3d", "XOM_ret_1d", "AVB_AvalonBay_zscore_60d", "QQQ_vol_20d", "JNJ_ret_1d", "EFFR_ret_1d", "ENB_EnbridgeInc_ret_1d", "ORCL_zscore_60d", "Retail_Sales_zscore_60d"], "is_new": true}, {"model_id": "new_h7_GLOBAL_RandomForest_N12_t3", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 7, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWM_Malaysia_zscore_60d", "MS_MorganStanley_zscore_60d", "Industrial_Production_zscore_60d", "EWY_Korea_ret_20d", "US6M_Rate_ret_20d", "EWC_Canada_zscore_60d", "FedFunds_zscore_60d", "PG_ret_20d", "XLB_Materials_zscore_60d", "PPL_PPL_ret_1d"], "is_new": true}, {"model_id": "new_h7_GLOBAL_RandomForest_N12_t4", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 7, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "AXP_Amex_vol_20d", "GD_GeneralDynamics_zscore_60d", "HD_zscore_60d", "Nikkei_Japan_zscore_60d", "LOW_Lowes_ret_5d", "JNJ_ret_1d", "US3M_Rate_zscore_60d", "IBEX_Spain_ret_20d", "TM_Telephone_vol_20d", "heston_ev_h3"], "is_new": true}, {"model_id": "new_h7_GLOBAL_RandomForest_N12_t5", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 7, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "BLK_BlackRock_zscore_60d", "TXN_vol_20d", "XOM_ret_1d", "HUM_Humana_ret_5d", "LOW_Lowes_ret_20d", "Michigan_Sentiment_ret_20d", "CPB_CampbellSoup_zscore_60d", "DOW_Price_zscore_60d", "Core_PCE_zscore_60d", "AMT_AmericanTower_ret_1d"], "is_new": true}, {"model_id": "new_h7_GLOBAL_RandomForest_N12_t6", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 7, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "gjr_condvar_h1", "US1Y_Rate_ret_5d", "TM_Telephone_vol_20d", "EFFR_ret_1d", "NWL_Newell_ret_20d", "Nikkei_Japan_vol_20d", "Core_PCE_zscore_60d", "QQQ_vol_20d", "BLK_BlackRock_zscore_60d", "CLX_Clorox_vol_20d"], "is_new": true}, {"model_id": "new_h7_GLOBAL_RandomForest_N12_t7", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 7, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "LLY_zscore_60d", "EWM_Malaysia_ret_1d", "AORD_AUS_zscore_60d", "3M_ret_5d", "EXC_Exelon_zscore_60d", "heston_var_ev_h5", "Nikkei_Japan_zscore_60d", "LOW_Lowes_ret_20d", "EFFR_vol_20d", "XLF_Fin_vol_20d"], "is_new": true}, {"model_id": "new_h7_GLOBAL_RandomForest_N15_t0", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 7, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "INTC_ret_1d", "EWM_Malaysia_zscore_60d", "LMT_LockheedMartin_vol_20d", "vix_mean_abs_ret_5d", "WTI_Oil_FRED_zscore_60d", "GE_ret_1d", "SLB_Schlumberger_ret_5d", "EOG_EOGResources_ret_5d", "SJM_JM_Smucker_ret_1d", "Brent_Oil_FRED_ret_20d", "heston_var_ev_h5", "MSTR_Bitcoin3_ret_20d", "US1Y_Rate_ret_5d"], "is_new": true}, {"model_id": "new_h7_GLOBAL_RandomForest_N15_t1", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 7, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "T_ret_1d", "LMT_LockheedMartin_vol_20d", "CPB_CampbellSoup_ret_20d", "MSTR_Bitcoin3_ret_20d", "Nikkei_Japan_zscore_60d", "EWC_Canada_zscore_60d", "CTAS_Cintas_vol_20d", "SO_SouthernCo_ret_5d", "FedFunds_zscore_60d", "US5Y_Rate_ret_5d", "US1Y_Rate_ret_5d", "EWM_Malaysia_vol_20d", "HD_ret_5d"], "is_new": true}, {"model_id": "new_h7_GLOBAL_RandomForest_N15_t2", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 7, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "SBUX_ret_5d", "AXP_Amex_vol_20d", "US30Y_Rate_ret_20d", "CPB_CampbellSoup_vol_20d", "SCHW_Schwab_ret_5d", "QQQ_vol_20d", "INTC_ret_1d", "EWL_Switzerland_vol_20d", "HD_zscore_60d", "EWH_HongKong_ret_5d", "EWL_Switzerland_zscore_60d", "HD_ret_20d", "XOM_ret_1d"], "is_new": true}, {"model_id": "new_h7_GLOBAL_RandomForest_N15_t3", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 7, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWM_Malaysia_ret_1d", "AMZN_ret_5d", "US7Y_Rate_ret_20d", "MO_AltriaMG_ret_1d", "DHR_vol_20d", "gjr_condvar_h1", "QQQ_vol_20d", "NOC_Northrop_ret_20d", "CPB_CampbellSoup_vol_20d", "EWQ_France_zscore_60d", "US1Y_Rate_ret_20d", "3M_ret_5d", "BLK_BlackRock_zscore_60d"], "is_new": true}, {"model_id": "new_h7_GLOBAL_RandomForest_N15_t4", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 7, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EXC_Exelon_zscore_60d", "PAYX_Paychex_zscore_60d", "GILD_Gilead_ret_20d", "EWQ_France_ret_20d", "TM_Telephone_ret_1d", "EFFR_ret_1d", "US1Y_Rate_ret_5d", "SJM_JM_Smucker_ret_5d", "DHR_ret_1d", "IYR_US_REIT2_zscore_60d", "US3M_Rate_zscore_60d", "XLF_Fin_vol_20d", "PPL_PPL_ret_1d"], "is_new": true}, {"model_id": "new_h7_GLOBAL_RandomForest_N15_t5", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 7, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "LLY_zscore_60d", "INTC_ret_5d", "AMGN_Amgen_ret_1d", "IBEX_Spain_ret_20d", "CPB_CampbellSoup_zscore_60d", "INTC_ret_1d", "T_ret_1d", "SBUX_vol_20d", "heston_var_ev_h7", "WTI_Oil_FRED_zscore_60d", "AORD_AUS_zscore_60d", "HD_ret_1d", "SLB_Schlumberger_ret_1d"], "is_new": true}, {"model_id": "new_h7_GLOBAL_RandomForest_N15_t6", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 7, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "PLD_Prologis_ret_5d", "spx_momentum_3d", "VVIX_ret_20d", "EQIX_Equinix_ret_5d", "HD_ret_1d", "EXC_Exelon_zscore_60d", "XLB_Materials_zscore_60d", "EWM_Malaysia_vol_20d", "XLK_Tech_zscore_60d", "DAX_Germany_zscore_60d", "EWA_Australia_zscore_60d", "CLX_Clorox_vol_20d", "US1Y_Rate_ret_5d"], "is_new": true}, {"model_id": "new_h7_GLOBAL_RandomForest_N15_t7", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 7, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "HangSeng_HK_ret_1d", "MS_MorganStanley_zscore_60d", "TXN_vol_20d", "HUM_Humana_ret_5d", "AXP_Amex_ret_20d", "SLB_Schlumberger_ret_1d", "US5Y_Rate_ret_5d", "3M_ret_5d", "XLY_Disc_vol_20d", "EXC_Exelon_zscore_60d", "EWC_Canada_zscore_60d", "FedFunds_zscore_60d", "SCHW_Schwab_ret_5d"], "is_new": true}, {"model_id": "new_h7_GLOBAL_RandomForest_N20_t0", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 7, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "SPY_zscore_60d", "DOW_Price_zscore_60d", "NVDA_vol_20d", "SLB_Schlumberger_ret_5d", "NWL_Newell_ret_20d", "EFFR_ret_1d", "HUM_Humana_ret_5d", "ITT_ITTInc_ret_5d", "EWA_Australia_zscore_60d", "Brent_Oil_FRED_ret_20d", "ASX_Australia_ret_5d", "US5Y_Rate_ret_5d", "Core_CPI_zscore_60d", "heston_var_ev_h3", "PAYX_Paychex_ret_20d", "CTAS_Cintas_vol_20d", "GE_ret_1d", "SBUX_ret_5d"], "is_new": true}, {"model_id": "new_h7_GLOBAL_RandomForest_N20_t1", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 7, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "XLY_Disc_vol_20d", "EWC_Canada_zscore_60d", "VOD_Vodafone_zscore_60d", "LOW_Lowes_ret_5d", "GD_GeneralDynamics_zscore_60d", "XOM_ret_20d", "US1Y_Rate_ret_5d", "hmm_p_stress", "EWS_Singapore_ret_5d", "ITT_ITTInc_ret_5d", "EWG_Germany_ret_20d", "EWM_Malaysia_vol_20d", "NEE_NextEra_ret_20d", "DHR_ret_1d", "DOW_Price_zscore_60d", "XOM_ret_1d", "Nikkei_Japan_zscore_60d", "SBUX_zscore_60d"], "is_new": true}, {"model_id": "new_h7_GLOBAL_RandomForest_N20_t2", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 7, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWC_Canada_zscore_60d", "Retail_Sales_zscore_60d", "spx_abs_ret_max_5d", "XLK_Tech_zscore_60d", "EWL_Switzerland_vol_20d", "EOG_EOGResources_vol_20d", "heston_var_ev_h7", "JNJ_ret_1d", "MO_AltriaMG_ret_1d", "FedFunds_zscore_60d", "US5Y_Rate_ret_5d", "AXP_Amex_ret_20d", "EXC_Exelon_ret_1d", "CTAS_Cintas_vol_20d", "TXN_vol_20d", "HangSeng_HK_vol_20d", "LOW_Lowes_ret_5d", "SLB_Schlumberger_ret_1d"], "is_new": true}, {"model_id": "new_h7_GLOBAL_RandomForest_N20_t3", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 7, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "US3M_Rate_zscore_60d", "Brent_Oil_FRED_ret_5d", "EWC_Canada_zscore_60d", "XLY_Disc_vol_20d", "MS_MorganStanley_ret_1d", "SCHW_Schwab_ret_5d", "PAYX_Paychex_ret_20d", "HangSeng_HK_vol_20d", "EWS_Singapore_ret_5d", "EXC_Exelon_ret_1d", "XOM_ret_20d", "CPB_CampbellSoup_vol_20d", "hmm_p_stress", "GD_GeneralDynamics_zscore_60d", "MSTR_Bitcoin3_ret_1d", "ORCL_zscore_60d", "Brent_Oil_FRED_ret_20d", "SJM_JM_Smucker_ret_1d"], "is_new": true}, {"model_id": "new_h7_GLOBAL_RandomForest_N20_t4", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 7, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "HangSeng_HK_ret_1d", "ITT_ITTInc_ret_5d", "EOG_EOGResources_ret_5d", "SBUX_ret_5d", "EWH_HongKong_ret_5d", "LMT_LockheedMartin_ret_1d", "DHR_vol_20d", "gjr_condvar_h1", "XLB_Materials_zscore_60d", "AMD_ret_5d", "BTI_BritishAmerican_ret_5d", "EWM_Malaysia_ret_1d", "HUM_Humana_ret_5d", "EWM_Malaysia_vol_20d", "MSTR_Bitcoin3_ret_5d", "VOD_Vodafone_zscore_60d", "HD_ret_5d", "LLY_zscore_60d"], "is_new": true}, {"model_id": "new_h7_GLOBAL_RandomForest_N20_t5", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 7, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "BA_ret_1d", "3M_vol_20d", "HangSeng_HK_ret_5d", "AORD_AUS_zscore_60d", "CPB_CampbellSoup_zscore_60d", "PAYX_Paychex_vol_20d", "PAYX_Paychex_zscore_60d", "NOC_Northrop_ret_20d", "PFE_ret_1d", "MRK_Merck_zscore_60d", "DHR_ret_1d", "ASX_Australia_ret_5d", "EWM_Malaysia_zscore_60d", "HangSeng_HK_vol_20d", "US5Y_Rate_ret_5d", "XLB_Materials_zscore_60d", "FedFunds_zscore_60d", "BTI_BritishAmerican_ret_20d"], "is_new": true}, {"model_id": "new_h7_GLOBAL_RandomForest_N20_t6", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 7, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "Michigan_Sentiment_ret_20d", "HD_zscore_60d", "HangSeng_HK_vol_20d", "AXP_Amex_vol_20d", "US1Y_Rate_ret_5d", "gjr_condvar_h1", "EWY_Korea_zscore_60d", "CPB_CampbellSoup_ret_20d", "AMGN_Amgen_ret_1d", "TM_Telephone_vol_20d", "MSTR_Bitcoin3_ret_1d", "PPL_PPL_ret_1d", "EFFR_ret_1d", "HD_ret_5d", "AORD_AUS_zscore_60d", "vix_acceleration_1d", "EWL_Switzerland_vol_20d", "IYR_US_REIT2_zscore_60d"], "is_new": true}, {"model_id": "new_h7_GLOBAL_RandomForest_N20_t7", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 7, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "T_ret_1d", "spx_vol_5d", "SJM_JM_Smucker_ret_1d", "TED_Spread_vol_20d", "PAYX_Paychex_ret_20d", "HUM_Humana_ret_5d", "heston_var_ev_h5", "Nikkei_Japan_zscore_60d", "SJM_JM_Smucker_ret_5d", "INTC_ret_1d", "CI_Cigna_vol_20d", "US3M_Rate_vol_20d", "EWQ_France_zscore_60d", "DIS_vol_20d", "VRP_ma5", "ORCL_vol_20d", "EWA_Australia_ret_1d", "ES_Evergy_ret_1d"], "is_new": true}, {"model_id": "new_h7_GLOBAL_RandomForest_N25_t0", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 7, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "XOM_ret_20d", "EFFR_vol_20d", "XLB_Materials_zscore_60d", "US1Y_Rate_ret_5d", "vix_acceleration_1d", "heston_var_ev_h5", "US3Y_Rate_ret_5d", "T_ret_1d", "SPY_zscore_60d", "EWQ_France_ret_20d", "SLB_Schlumberger_ret_5d", "3M_vol_20d", "GILD_Gilead_ret_20d", "CTAS_Cintas_vol_20d", "NEE_NextEra_ret_20d", "spx_vol_5d", "EWG_Germany_vol_20d", "PCAR_PaccarInc_ret_5d", "NFCI_ret_5d", "PAYX_Paychex_ret_20d", "vix_mean_abs_ret_5d", "SLB_Schlumberger_ret_1d", "Nikkei_Japan_zscore_60d"], "is_new": true}, {"model_id": "new_h7_GLOBAL_RandomForest_N25_t1", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 7, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "SJM_JM_Smucker_ret_1d", "IYM_BasicMaterials_ret_20d", "US1Y_Rate_ret_5d", "GD_GeneralDynamics_zscore_60d", "US6M_Rate_ret_20d", "EWL_Switzerland_zscore_60d", "DAX_Germany_vol_20d", "US7Y_Rate_ret_20d", "SBUX_zscore_60d", "HD_ret_20d", "EWY_Korea_ret_20d", "MSTR_Bitcoin3_ret_5d", "EFFR_ret_1d", "vix_mean_abs_ret_5d", "CPB_CampbellSoup_ret_5d", "EXC_Exelon_ret_1d", "Core_PCE_zscore_60d", "heston_var_ev_h3", "EOG_EOGResources_vol_20d", "DIS_vol_20d", "DOW_Price_zscore_60d", "T_ret_1d", "EWQ_France_zscore_60d"], "is_new": true}, {"model_id": "new_h7_GLOBAL_RandomForest_N25_t2", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 7, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "TED_Spread_zscore_60d", "DIS_vol_20d", "TED_Spread_vol_20d", "SJM_JM_Smucker_ret_5d", "IYR_US_REIT2_zscore_60d", "EWH_HongKong_ret_5d", "SBUX_ret_5d", "XOM_ret_20d", "US1Y_Rate_ret_20d", "EWY_Korea_ret_20d", "vix_mean_abs_ret_5d", "CPB_CampbellSoup_zscore_60d", "HangSeng_HK_ret_1d", "US6M_Rate_ret_20d", "M_Macys_vol_20d", "EQIX_Equinix_ret_5d", "EWQ_France_zscore_60d", "HUM_Humana_ret_5d", "AMD_ret_1d", "EFFR_vol_20d", "Brent_Oil_FRED_ret_5d", "MSTR_Bitcoin3_ret_20d", "EXC_Exelon_ret_1d"], "is_new": true}, {"model_id": "new_h7_GLOBAL_RandomForest_N25_t3", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 7, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "CPB_CampbellSoup_ret_5d", "XLK_Tech_zscore_60d", "EWY_Korea_ret_20d", "GILD_Gilead_ret_20d", "DOW_Price_zscore_60d", "XLB_Materials_zscore_60d", "TGT_Target_zscore_60d", "NOC_Northrop_ret_20d", "DAX_Germany_zscore_60d", "HangSeng_HK_ret_5d", "heston_var_ev_h7", "DE_Deere_ret_5d", "XOM_ret_1d", "EQR_Equity_ret_1d", "ES_Evergy_ret_1d", "US3M_Rate_zscore_60d", "EXC_Exelon_ret_1d", "EWM_Malaysia_ret_1d", "XOM_ret_20d", "EMR_Emerson_ret_20d", "PG_ret_20d", "CPB_CampbellSoup_zscore_60d", "Brent_Oil_FRED_ret_5d"], "is_new": true}, {"model_id": "new_h7_GLOBAL_RandomForest_N25_t4", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 7, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "SBUX_zscore_60d", "SPY_zscore_60d", "Brent_Oil_FRED_ret_20d", "HD_ret_20d", "US1Y_Rate_ret_20d", "CTAS_Cintas_vol_20d", "T_ret_1d", "BLK_BlackRock_zscore_60d", "EQR_Equity_ret_1d", "ASX_Australia_vol_20d", "MO_AltriaMG_ret_1d", "spx_momentum_3d", "AMD_ret_5d", "IYR_US_REIT2_zscore_60d", "SO_SouthernCo_ret_5d", "DE_Deere_ret_5d", "SCHW_Schwab_ret_5d", "BTI_BritishAmerican_ret_5d", "HD_ret_1d", "XOM_ret_1d", "EXC_Exelon_zscore_60d", "Core_CPI_zscore_60d", "spx_abs_ret_max_5d"], "is_new": true}, {"model_id": "new_h7_GLOBAL_RandomForest_N25_t5", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 7, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "Michigan_Sentiment_ret_20d", "EWQ_France_zscore_60d", "SJM_JM_Smucker_ret_5d", "AMGN_Amgen_ret_1d", "HUM_Humana_ret_5d", "AORD_AUS_zscore_60d", "spx_momentum_3d", "HangSeng_HK_vol_20d", "LUV_SouthwestAir_ret_5d", "heston_var_ev_h3", "EQR_Equity_ret_1d", "MRK_Merck_zscore_60d", "DOW_Price_zscore_60d", "NEE_NextEra_ret_20d", "INTC_ret_1d", "HD_ret_1d", "MS_MorganStanley_ret_1d", "BDX_Becton_Dickinson_ret_20d", "vix_mean_abs_ret_5d", "SBUX_zscore_60d", "EQIX_Equinix_ret_5d", "XLY_Disc_vol_20d", "spx_vol_5d"], "is_new": true}, {"model_id": "new_h7_GLOBAL_RandomForest_N25_t6", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 7, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "MRK_Merck_zscore_60d", "TED_Spread_vol_20d", "AMD_ret_1d", "US3Y_Rate_ret_5d", "Nikkei_Japan_zscore_60d", "XLV_Health_zscore_60d", "NFCI_ret_5d", "EWQ_France_zscore_60d", "3M_ret_5d", "spx_momentum_3d", "INTC_ret_1d", "M_Macys_vol_20d", "QQQ_vol_20d", "LOW_Lowes_ret_5d", "ITT_ITTInc_ret_5d", "BA_ret_1d", "EWG_Germany_ret_20d", "PAYX_Paychex_ret_20d", "XOM_ret_20d", "HangSeng_HK_ret_5d", "SBUX_vol_20d", "SO_SouthernCo_ret_5d", "US6M_Rate_ret_20d"], "is_new": true}, {"model_id": "new_h7_GLOBAL_RandomForest_N25_t7", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 7, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "3M_vol_20d", "Retail_Sales_zscore_60d", "HangSeng_HK_ret_1d", "ORCL_vol_20d", "EMR_Emerson_ret_20d", "SPY_zscore_60d", "Michigan_Sentiment_ret_20d", "US3Y_Rate_ret_5d", "CPB_CampbellSoup_vol_20d", "DAX_Germany_vol_20d", "BDX_Becton_Dickinson_ret_20d", "IYM_BasicMaterials_ret_20d", "vix_mean_abs_ret_5d", "NOC_Northrop_ret_20d", "TM_Telephone_ret_1d", "EWJ_Japan_vol_20d", "LUV_SouthwestAir_ret_5d", "SJM_JM_Smucker_ret_5d", "vix_acceleration_1d", "SCHW_Schwab_ret_5d", "MSTR_Bitcoin3_ret_5d", "MS_MorganStanley_zscore_60d", "EWA_Australia_zscore_60d"], "is_new": true}, {"model_id": "new_h7_GLOBAL_RandomForest_N30_t0", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 7, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWY_Korea_zscore_60d", "heston_var_ev_h7", "BA_ret_1d", "MRK_Merck_zscore_60d", "EWY_Korea_ret_20d", "SBUX_zscore_60d", "NVDA_vol_20d", "PAYX_Paychex_ret_20d", "ASX_Australia_ret_5d", "EXC_Exelon_ret_1d", "CI_Cigna_vol_20d", "INTC_ret_5d", "Core_CPI_zscore_60d", "AMD_ret_1d", "M_Macys_vol_20d", "CLX_Clorox_vol_20d", "EQIX_Equinix_ret_5d", "IYR_US_REIT2_zscore_60d", "DHR_vol_20d", "BLK_BlackRock_zscore_60d", "Michigan_Sentiment_ret_20d", "VVIX_ret_20d", "INTC_ret_1d", "PAYX_Paychex_vol_20d", "EWM_Malaysia_ret_1d", "GE_ret_1d", "DAX_Germany_vol_20d", "XLF_Fin_vol_20d"], "is_new": true}, {"model_id": "new_h7_GLOBAL_RandomForest_N30_t1", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 7, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "GE_ret_1d", "Industrial_Production_zscore_60d", "SBUX_zscore_60d", "US3M_Rate_zscore_60d", "IWM_SmallCap_vol_20d", "M_Macys_vol_20d", "AXP_Amex_ret_20d", "LLY_zscore_60d", "NVDA_vol_20d", "ASX_Australia_ret_5d", "Retail_Sales_zscore_60d", "EWQ_France_zscore_60d", "PLD_Prologis_ret_5d", "ORCL_zscore_60d", "CI_Cigna_vol_20d", "Core_PCE_zscore_60d", "SLB_Schlumberger_ret_1d", "TM_Telephone_vol_20d", "EQR_Equity_ret_1d", "EWY_Korea_zscore_60d", "EMR_Emerson_ret_20d", "AORD_AUS_zscore_60d", "INTC_ret_5d", "DAX_Germany_vol_20d", "WTI_Oil_FRED_zscore_60d", "MRK_Merck_zscore_60d", "EWQ_France_ret_20d", "VRP_ma5"], "is_new": true}, {"model_id": "new_h7_GLOBAL_RandomForest_N30_t2", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 7, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "US5Y_Rate_ret_5d", "EWY_Korea_zscore_60d", "CPB_CampbellSoup_zscore_60d", "PAYX_Paychex_vol_20d", "EWM_Malaysia_zscore_60d", "QQQ_vol_20d", "CPB_CampbellSoup_vol_20d", "HD_ret_5d", "EXC_Exelon_ret_1d", "ES_Evergy_ret_1d", "Core_CPI_zscore_60d", "heston_var_ev_h5", "AMT_AmericanTower_ret_1d", "EFFR_vol_20d", "AVB_AvalonBay_zscore_60d", "VVIX_ret_20d", "EOG_EOGResources_ret_5d", "BTI_BritishAmerican_ret_20d", "BDX_Becton_Dickinson_ret_20d", "DE_Deere_ret_5d", "SJM_JM_Smucker_ret_5d", "EWY_Korea_ret_20d", "EXC_Exelon_zscore_60d", "SO_SouthernCo_ret_5d", "NVDA_vol_20d", "EWA_Australia_ret_1d", "MRK_Merck_zscore_60d", "INTC_ret_5d"], "is_new": true}, {"model_id": "new_h7_GLOBAL_RandomForest_N30_t3", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 7, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWA_Australia_ret_1d", "XOM_ret_20d", "PCAR_PaccarInc_ret_5d", "EQIX_Equinix_ret_5d", "VVIX_ret_20d", "INTC_ret_5d", "HD_zscore_60d", "BTI_BritishAmerican_ret_20d", "ENB_EnbridgeInc_ret_1d", "ITT_ITTInc_ret_5d", "EOG_EOGResources_vol_20d", "CPB_CampbellSoup_ret_20d", "CCI_CrownCastle_vol_20d", "Core_PCE_zscore_60d", "MS_MorganStanley_zscore_60d", "US7Y_Rate_ret_20d", "ORCL_zscore_60d", "vix_mean_abs_ret_5d", "XLV_Health_zscore_60d", "M_Macys_vol_20d", "MO_AltriaMG_ret_1d", "GE_ret_1d", "3M_ret_5d", "TED_Spread_vol_20d", "ASX_Australia_ret_5d", "AVB_AvalonBay_zscore_60d", "HD_ret_1d", "EWC_Canada_zscore_60d"], "is_new": true}, {"model_id": "new_h7_GLOBAL_RandomForest_N30_t4", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 7, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "NEE_NextEra_ret_20d", "T10Y2Y_Spread_ret_5d", "HD_ret_5d", "CI_Cigna_vol_20d", "US1Y_Rate_ret_20d", "Nikkei_Japan_zscore_60d", "AORD_AUS_zscore_60d", "INTC_ret_5d", "EWG_Germany_ret_20d", "XLV_Health_zscore_60d", "AVB_AvalonBay_zscore_60d", "AXP_Amex_ret_20d", "XOM_ret_1d", "3M_ret_5d", "LOW_Lowes_ret_5d", "US30Y_Rate_ret_20d", "GILD_Gilead_ret_20d", "TED_Spread_vol_20d", "CMCSA_ret_1d", "EWS_Singapore_ret_5d", "NFCI_ret_5d", "LLY_zscore_60d", "NVDA_vol_20d", "heston_var_ev_h5", "VRP_ma5", "JNJ_ret_1d", "vix_mean_abs_ret_5d", "MS_MorganStanley_zscore_60d"], "is_new": true}, {"model_id": "new_h7_GLOBAL_RandomForest_N30_t5", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 7, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "US1Y_Rate_ret_20d", "AMD_ret_1d", "EMR_Emerson_ret_20d", "IYM_BasicMaterials_ret_20d", "EXC_Exelon_ret_1d", "EWJ_Japan_vol_20d", "PLD_Prologis_ret_5d", "EWH_HongKong_ret_5d", "CPB_CampbellSoup_vol_20d", "MS_MorganStanley_ret_5d", "PAYX_Paychex_vol_20d", "NVDA_vol_20d", "XLK_Tech_zscore_60d", "SJM_JM_Smucker_ret_5d", "PG_ret_20d", "INTC_ret_1d", "DAX_Germany_vol_20d", "IBEX_Spain_ret_20d", "MS_MorganStanley_zscore_60d", "FedFunds_zscore_60d", "WTI_Oil_FRED_zscore_60d", "AMT_AmericanTower_ret_1d", "VRP_ma5", "ES_Evergy_ret_1d", "XOM_ret_1d", "Brent_Oil_FRED_ret_20d", "ASX_Australia_ret_5d", "GD_GeneralDynamics_zscore_60d"], "is_new": true}, {"model_id": "new_h7_GLOBAL_RandomForest_N30_t6", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 7, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "vix_mean_abs_ret_5d", "PCAR_PaccarInc_ret_5d", "HD_ret_5d", "LMT_LockheedMartin_ret_1d", "HangSeng_HK_ret_5d", "Nikkei_Japan_zscore_60d", "XLB_Materials_zscore_60d", "M_Macys_vol_20d", "Michigan_Sentiment_ret_20d", "EQIX_Equinix_ret_5d", "NWL_Newell_ret_20d", "HUM_Humana_ret_5d", "TM_Telephone_ret_1d", "US3Y_Rate_ret_5d", "Retail_Sales_zscore_60d", "EWG_Germany_ret_20d", "3M_vol_20d", "CPB_CampbellSoup_vol_20d", "ES_Evergy_ret_1d", "EWH_HongKong_ret_5d", "BTI_BritishAmerican_ret_20d", "US3M_Rate_zscore_60d", "LUV_SouthwestAir_ret_5d", "MS_MorganStanley_zscore_60d", "MS_MorganStanley_ret_1d", "EMR_Emerson_ret_20d", "US6M_Rate_ret_20d", "QQQ_vol_20d"], "is_new": true}, {"model_id": "new_h7_GLOBAL_RandomForest_N30_t7", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 7, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "MS_MorganStanley_ret_1d", "AMT_AmericanTower_ret_1d", "NEE_NextEra_ret_20d", "HD_ret_1d", "US3Y_Rate_ret_5d", "MS_MorganStanley_ret_5d", "heston_var_ev_h7", "DHR_ret_1d", "Core_PCE_zscore_60d", "US3M_Rate_vol_20d", "SBUX_ret_5d", "Nikkei_Japan_zscore_60d", "LMT_LockheedMartin_ret_1d", "SLB_Schlumberger_ret_1d", "TED_Spread_vol_20d", "M_Macys_vol_20d", "AMZN_ret_5d", "T_ret_1d", "CTAS_Cintas_vol_20d", "EOG_EOGResources_ret_5d", "XOM_ret_1d", "heston_var_ev_h3", "CI_Cigna_vol_20d", "EWL_Switzerland_vol_20d", "EWM_Malaysia_vol_20d", "T10Y2Y_Spread_ret_5d", "SJM_JM_Smucker_ret_1d", "QQQ_vol_20d"], "is_new": true}, {"model_id": "new_h7_GLOBAL_LogisticRegression_N5_t0", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 7, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "BTI_BritishAmerican_ret_5d", "MSTR_Bitcoin3_ret_20d", "US1Y_Rate_ret_20d"], "is_new": true}, {"model_id": "new_h7_GLOBAL_LogisticRegression_N5_t1", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 7, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "LUV_SouthwestAir_ret_5d", "SBUX_ret_5d", "EWA_Australia_ret_1d"], "is_new": true}, {"model_id": "new_h7_GLOBAL_LogisticRegression_N5_t2", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 7, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "CCI_CrownCastle_vol_20d", "XLB_Materials_zscore_60d", "AVB_AvalonBay_zscore_60d"], "is_new": true}, {"model_id": "new_h7_GLOBAL_LogisticRegression_N5_t3", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 7, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "AMD_ret_5d", "ASX_Australia_ret_5d", "MS_MorganStanley_ret_5d"], "is_new": true}, {"model_id": "new_h7_GLOBAL_LogisticRegression_N5_t4", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 7, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "SLB_Schlumberger_ret_1d", "AMD_ret_1d", "EWL_Switzerland_zscore_60d"], "is_new": true}, {"model_id": "new_h7_GLOBAL_LogisticRegression_N5_t5", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 7, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "heston_var_ev_h7", "PLD_Prologis_ret_5d", "MO_AltriaMG_ret_1d"], "is_new": true}, {"model_id": "new_h7_GLOBAL_LogisticRegression_N5_t6", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 7, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "heston_var_ev_h5", "EFFR_ret_1d", "Nikkei_Japan_vol_20d"], "is_new": true}, {"model_id": "new_h7_GLOBAL_LogisticRegression_N5_t7", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 7, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "ORCL_zscore_60d", "US7Y_Rate_ret_20d", "T_ret_1d"], "is_new": true}, {"model_id": "new_h7_GLOBAL_LogisticRegression_N8_t0", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 7, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWY_Korea_ret_20d", "CPB_CampbellSoup_ret_20d", "IYM_BasicMaterials_ret_20d", "spx_vol_5d", "BLK_BlackRock_zscore_60d", "NFCI_ret_5d"], "is_new": true}, {"model_id": "new_h7_GLOBAL_LogisticRegression_N8_t1", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 7, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "GE_ret_1d", "EOG_EOGResources_ret_5d", "AMD_ret_1d", "PAYX_Paychex_vol_20d", "CCI_CrownCastle_vol_20d", "EOG_EOGResources_vol_20d"], "is_new": true}, {"model_id": "new_h7_GLOBAL_LogisticRegression_N8_t2", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 7, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "Core_PCE_zscore_60d", "EFFR_vol_20d", "PAYX_Paychex_zscore_60d", "AMZN_ret_5d", "T10Y2Y_Spread_ret_5d", "EWM_Malaysia_zscore_60d"], "is_new": true}, {"model_id": "new_h7_GLOBAL_LogisticRegression_N8_t3", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 7, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWC_Canada_zscore_60d", "HangSeng_HK_ret_5d", "SJM_JM_Smucker_ret_5d", "PAYX_Paychex_zscore_60d", "CPB_CampbellSoup_ret_20d", "EWG_Germany_ret_20d"], "is_new": true}, {"model_id": "new_h7_GLOBAL_LogisticRegression_N8_t4", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 7, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "LMT_LockheedMartin_vol_20d", "spx_abs_ret_max_5d", "PCAR_PaccarInc_ret_5d", "gjr_condvar_h1", "CI_Cigna_vol_20d", "EWH_HongKong_ret_5d"], "is_new": true}, {"model_id": "new_h7_GLOBAL_LogisticRegression_N8_t5", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 7, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "DHR_vol_20d", "EFFR_ret_1d", "MRK_Merck_zscore_60d", "SJM_JM_Smucker_ret_1d", "ORCL_vol_20d", "IWM_SmallCap_vol_20d"], "is_new": true}, {"model_id": "new_h7_GLOBAL_LogisticRegression_N8_t6", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 7, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "HD_ret_1d", "XLF_Fin_vol_20d", "CPB_CampbellSoup_ret_5d", "HD_ret_5d", "EFFR_vol_20d", "ES_Evergy_ret_1d"], "is_new": true}, {"model_id": "new_h7_GLOBAL_LogisticRegression_N8_t7", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 7, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "US7Y_Rate_ret_20d", "BDX_Becton_Dickinson_ret_20d", "JNJ_ret_1d", "PG_ret_20d", "EWY_Korea_zscore_60d", "Core_PCE_zscore_60d"], "is_new": true}, {"model_id": "new_h7_GLOBAL_LogisticRegression_N10_t0", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 7, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "heston_var_ev_h7", "LLY_zscore_60d", "HD_ret_5d", "HangSeng_HK_ret_5d", "AVB_AvalonBay_zscore_60d", "VRP_ma5", "US1Y_Rate_ret_5d", "Retail_Sales_zscore_60d"], "is_new": true}, {"model_id": "new_h7_GLOBAL_LogisticRegression_N10_t1", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 7, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "Core_CPI_zscore_60d", "XLF_Fin_vol_20d", "SLB_Schlumberger_ret_5d", "VVIX_ret_20d", "3M_vol_20d", "Industrial_Production_zscore_60d", "DIS_vol_20d", "NEE_NextEra_ret_20d"], "is_new": true}, {"model_id": "new_h7_GLOBAL_LogisticRegression_N10_t2", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 7, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "Brent_Oil_FRED_ret_5d", "AMD_ret_1d", "NFCI_ret_5d", "VOD_Vodafone_zscore_60d", "JNJ_ret_1d", "MS_MorganStanley_ret_1d", "ES_Evergy_ret_1d", "IWM_SmallCap_vol_20d"], "is_new": true}, {"model_id": "new_h7_GLOBAL_LogisticRegression_N10_t3", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 7, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "TED_Spread_zscore_60d", "US7Y_Rate_ret_20d", "ES_Evergy_ret_1d", "MS_MorganStanley_ret_5d", "XLY_Disc_vol_20d", "EWQ_France_ret_20d", "Brent_Oil_FRED_ret_5d", "ORCL_zscore_60d"], "is_new": true}, {"model_id": "new_h7_GLOBAL_LogisticRegression_N10_t4", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 7, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWM_Malaysia_zscore_60d", "AORD_AUS_zscore_60d", "MSTR_Bitcoin3_ret_1d", "DAX_Germany_vol_20d", "EMR_Emerson_ret_20d", "CMCSA_ret_1d", "EQIX_Equinix_ret_5d", "NFCI_ret_5d"], "is_new": true}, {"model_id": "new_h7_GLOBAL_LogisticRegression_N10_t5", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 7, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWL_Switzerland_vol_20d", "CTAS_Cintas_vol_20d", "US5Y_Rate_ret_5d", "ORCL_vol_20d", "NWL_Newell_ret_20d", "AVB_AvalonBay_zscore_60d", "AXP_Amex_ret_20d", "LLY_zscore_60d"], "is_new": true}, {"model_id": "new_h7_GLOBAL_LogisticRegression_N10_t6", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 7, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "LOW_Lowes_ret_20d", "EWS_Singapore_ret_5d", "CPB_CampbellSoup_ret_5d", "CPB_CampbellSoup_zscore_60d", "AORD_AUS_zscore_60d", "EFFR_ret_1d", "US7Y_Rate_ret_20d", "3M_vol_20d"], "is_new": true}, {"model_id": "new_h7_GLOBAL_LogisticRegression_N10_t7", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 7, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "3M_vol_20d", "SBUX_ret_5d", "IBEX_Spain_ret_20d", "DAX_Germany_vol_20d", "XLY_Disc_vol_20d", "spx_vol_5d", "US30Y_Rate_ret_20d", "EWQ_France_zscore_60d"], "is_new": true}, {"model_id": "new_h7_GLOBAL_LogisticRegression_N12_t0", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 7, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "DAX_Germany_vol_20d", "TED_Spread_zscore_60d", "SBUX_ret_5d", "US3M_Rate_vol_20d", "TXN_vol_20d", "ORCL_zscore_60d", "TM_Telephone_ret_1d", "BTI_BritishAmerican_ret_5d", "Brent_Oil_FRED_ret_5d", "CPB_CampbellSoup_vol_20d"], "is_new": true}, {"model_id": "new_h7_GLOBAL_LogisticRegression_N12_t1", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 7, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "XOM_ret_20d", "US1Y_Rate_ret_20d", "ES_Evergy_ret_1d", "Brent_Oil_FRED_ret_5d", "ASX_Australia_ret_5d", "NWL_Newell_ret_20d", "EFFR_vol_20d", "NEE_NextEra_ret_20d", "CCI_CrownCastle_vol_20d", "DAX_Germany_zscore_60d"], "is_new": true}, {"model_id": "new_h7_GLOBAL_LogisticRegression_N12_t2", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 7, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "SJM_JM_Smucker_ret_5d", "vix_mean_abs_ret_5d", "XOM_ret_1d", "EOG_EOGResources_ret_5d", "MSTR_Bitcoin3_ret_20d", "AORD_AUS_zscore_60d", "DIS_vol_20d", "M_Macys_vol_20d", "EMR_Emerson_ret_20d", "HUM_Humana_ret_5d"], "is_new": true}, {"model_id": "new_h7_GLOBAL_LogisticRegression_N12_t3", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 7, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "VOD_Vodafone_zscore_60d", "INTC_ret_5d", "SJM_JM_Smucker_ret_5d", "BDX_Becton_Dickinson_ret_20d", "Nikkei_Japan_zscore_60d", "XLF_Fin_vol_20d", "LOW_Lowes_ret_5d", "EWJ_Japan_vol_20d", "AMD_ret_1d", "DHR_vol_20d"], "is_new": true}, {"model_id": "new_h7_GLOBAL_LogisticRegression_N12_t4", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 7, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "Industrial_Production_zscore_60d", "HangSeng_HK_ret_5d", "Core_PCE_zscore_60d", "EWM_Malaysia_zscore_60d", "US3M_Rate_vol_20d", "EWG_Germany_ret_20d", "SO_SouthernCo_ret_5d", "LMT_LockheedMartin_vol_20d", "AORD_AUS_zscore_60d", "EWJ_Japan_vol_20d"], "is_new": true}, {"model_id": "new_h7_GLOBAL_LogisticRegression_N12_t5", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 7, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "ITT_ITTInc_ret_5d", "HangSeng_HK_ret_5d", "US7Y_Rate_ret_20d", "CPB_CampbellSoup_zscore_60d", "LOW_Lowes_ret_5d", "SJM_JM_Smucker_ret_1d", "CTAS_Cintas_vol_20d", "EXC_Exelon_zscore_60d", "AMD_ret_5d", "GE_ret_1d"], "is_new": true}, {"model_id": "new_h7_GLOBAL_LogisticRegression_N12_t6", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 7, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "SBUX_zscore_60d", "MS_MorganStanley_zscore_60d", "BLK_BlackRock_zscore_60d", "spx_vol_5d", "NFCI_ret_5d", "QQQ_vol_20d", "PFE_ret_1d", "EWQ_France_zscore_60d", "EWJ_Japan_vol_20d", "3M_vol_20d"], "is_new": true}, {"model_id": "new_h7_GLOBAL_LogisticRegression_N12_t7", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 7, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "NVDA_vol_20d", "EQR_Equity_ret_1d", "CMCSA_ret_1d", "DHR_vol_20d", "MSTR_Bitcoin3_ret_20d", "CPB_CampbellSoup_ret_20d", "EWA_Australia_ret_1d", "PLD_Prologis_ret_5d", "CI_Cigna_vol_20d", "MS_MorganStanley_zscore_60d"], "is_new": true}, {"model_id": "new_h7_GLOBAL_LogisticRegression_N15_t0", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 7, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "GD_GeneralDynamics_zscore_60d", "US30Y_Rate_ret_20d", "PCAR_PaccarInc_ret_5d", "XOM_ret_20d", "EWC_Canada_zscore_60d", "LLY_zscore_60d", "PG_ret_20d", "AMZN_ret_5d", "AVB_AvalonBay_zscore_60d", "ASX_Australia_vol_20d", "BTI_BritishAmerican_ret_5d", "CPB_CampbellSoup_ret_20d", "Nikkei_Japan_zscore_60d"], "is_new": true}, {"model_id": "new_h7_GLOBAL_LogisticRegression_N15_t1", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 7, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "CPB_CampbellSoup_ret_5d", "US30Y_Rate_ret_20d", "ORCL_zscore_60d", "SLB_Schlumberger_ret_5d", "NWL_Newell_ret_20d", "CMCSA_ret_1d", "heston_var_ev_h5", "HD_ret_1d", "US1Y_Rate_ret_20d", "EWG_Germany_vol_20d", "HD_ret_20d", "spx_momentum_3d", "Nikkei_Japan_vol_20d"], "is_new": true}, {"model_id": "new_h7_GLOBAL_LogisticRegression_N15_t2", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 7, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "PAYX_Paychex_ret_20d", "LUV_SouthwestAir_ret_5d", "SBUX_vol_20d", "EWA_Australia_zscore_60d", "Brent_Oil_FRED_ret_5d", "IBEX_Spain_ret_20d", "heston_var_ev_h5", "EWQ_France_ret_20d", "AMD_ret_1d", "spx_vol_5d", "CCI_CrownCastle_vol_20d", "PPL_PPL_ret_1d", "BTI_BritishAmerican_ret_20d"], "is_new": true}, {"model_id": "new_h7_GLOBAL_LogisticRegression_N15_t3", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 7, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "SJM_JM_Smucker_ret_5d", "EWA_Australia_ret_1d", "TED_Spread_zscore_60d", "LMT_LockheedMartin_ret_1d", "DHR_ret_1d", "PAYX_Paychex_vol_20d", "PFE_ret_1d", "vix_acceleration_1d", "DHR_vol_20d", "TXN_vol_20d", "ES_Evergy_ret_1d", "SO_SouthernCo_ret_5d", "US3M_Rate_zscore_60d"], "is_new": true}, {"model_id": "new_h7_GLOBAL_LogisticRegression_N15_t4", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 7, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "CPB_CampbellSoup_ret_20d", "DE_Deere_vol_20d", "EMR_Emerson_ret_20d", "GD_GeneralDynamics_zscore_60d", "HD_ret_20d", "XLF_Fin_vol_20d", "HD_ret_1d", "T10Y2Y_Spread_ret_5d", "SLB_Schlumberger_ret_1d", "SBUX_zscore_60d", "BDX_Becton_Dickinson_ret_20d", "VRP_ma5", "EWJ_Japan_vol_20d"], "is_new": true}, {"model_id": "new_h7_GLOBAL_LogisticRegression_N15_t5", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 7, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "heston_var_ev_h3", "GD_GeneralDynamics_zscore_60d", "SLB_Schlumberger_ret_1d", "HD_ret_20d", "ORCL_vol_20d", "VVIX_ret_20d", "3M_vol_20d", "EWC_Canada_zscore_60d", "hmm_p_stress", "MSTR_Bitcoin3_ret_1d", "vix_acceleration_1d", "EWS_Singapore_ret_5d", "MS_MorganStanley_ret_1d"], "is_new": true}, {"model_id": "new_h7_GLOBAL_LogisticRegression_N15_t6", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 7, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "PAYX_Paychex_zscore_60d", "AMGN_Amgen_ret_1d", "GD_GeneralDynamics_zscore_60d", "ENB_EnbridgeInc_ret_1d", "TXN_vol_20d", "PAYX_Paychex_ret_20d", "Core_PCE_zscore_60d", "BTI_BritishAmerican_ret_20d", "ES_Evergy_ret_1d", "PG_ret_20d", "SLB_Schlumberger_ret_5d", "BTI_BritishAmerican_ret_5d", "MSTR_Bitcoin3_ret_20d"], "is_new": true}, {"model_id": "new_h7_GLOBAL_LogisticRegression_N15_t7", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 7, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "SJM_JM_Smucker_ret_5d", "TM_Telephone_ret_1d", "heston_var_ev_h3", "US1Y_Rate_ret_5d", "CPB_CampbellSoup_vol_20d", "EWJ_Japan_vol_20d", "VVIX_ret_20d", "US3M_Rate_vol_20d", "AXP_Amex_ret_20d", "Core_CPI_zscore_60d", "PAYX_Paychex_ret_20d", "SLB_Schlumberger_ret_5d", "MRK_Merck_zscore_60d"], "is_new": true}, {"model_id": "new_h7_GLOBAL_LogisticRegression_N20_t0", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 7, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "PCAR_PaccarInc_ret_5d", "EFFR_vol_20d", "ASX_Australia_vol_20d", "AMT_AmericanTower_ret_1d", "PAYX_Paychex_zscore_60d", "CLX_Clorox_vol_20d", "gjr_condvar_h1", "AXP_Amex_ret_20d", "SJM_JM_Smucker_ret_5d", "BTI_BritishAmerican_ret_5d", "EWM_Malaysia_vol_20d", "EWY_Korea_ret_20d", "EOG_EOGResources_vol_20d", "HD_ret_20d", "Core_CPI_zscore_60d", "DOW_Price_zscore_60d", "INTC_ret_5d", "US3M_Rate_vol_20d"], "is_new": true}, {"model_id": "new_h7_GLOBAL_LogisticRegression_N20_t1", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 7, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWQ_France_zscore_60d", "HangSeng_HK_vol_20d", "TM_Telephone_ret_1d", "LUV_SouthwestAir_ret_5d", "EWA_Australia_ret_1d", "MS_MorganStanley_ret_5d", "TED_Spread_vol_20d", "CPB_CampbellSoup_ret_5d", "US3M_Rate_zscore_60d", "BDX_Becton_Dickinson_ret_20d", "US7Y_Rate_ret_20d", "SBUX_zscore_60d", "PFE_ret_1d", "Nikkei_Japan_vol_20d", "QQQ_vol_20d", "HD_ret_5d", "NFCI_ret_5d", "PAYX_Paychex_zscore_60d"], "is_new": true}, {"model_id": "new_h7_GLOBAL_LogisticRegression_N20_t2", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 7, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "DIS_vol_20d", "LUV_SouthwestAir_ret_5d", "AVB_AvalonBay_zscore_60d", "US3Y_Rate_ret_5d", "EWS_Singapore_ret_5d", "PPL_PPL_ret_1d", "BLK_BlackRock_zscore_60d", "ASX_Australia_vol_20d", "XLB_Materials_zscore_60d", "CPB_CampbellSoup_zscore_60d", "CCI_CrownCastle_vol_20d", "Core_CPI_zscore_60d", "NEE_NextEra_ret_20d", "INTC_ret_1d", "EWC_Canada_zscore_60d", "QQQ_vol_20d", "Retail_Sales_zscore_60d", "IYM_BasicMaterials_ret_20d"], "is_new": true}, {"model_id": "new_h7_GLOBAL_LogisticRegression_N20_t3", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 7, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "AVB_AvalonBay_zscore_60d", "MS_MorganStanley_zscore_60d", "CPB_CampbellSoup_vol_20d", "MSTR_Bitcoin3_ret_5d", "spx_vol_5d", "EWG_Germany_vol_20d", "ES_Evergy_ret_1d", "SJM_JM_Smucker_ret_5d", "US5Y_Rate_ret_5d", "NFCI_ret_5d", "SBUX_zscore_60d", "PAYX_Paychex_ret_20d", "MO_AltriaMG_ret_1d", "SJM_JM_Smucker_ret_1d", "XLF_Fin_vol_20d", "TM_Telephone_ret_1d", "LOW_Lowes_ret_20d", "EXC_Exelon_ret_1d"], "is_new": true}, {"model_id": "new_h7_GLOBAL_LogisticRegression_N20_t4", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 7, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "US3Y_Rate_ret_5d", "VOD_Vodafone_zscore_60d", "CCI_CrownCastle_vol_20d", "US1Y_Rate_ret_5d", "XLF_Fin_vol_20d", "CPB_CampbellSoup_ret_5d", "ORCL_vol_20d", "EFFR_vol_20d", "CMCSA_ret_1d", "LUV_SouthwestAir_ret_5d", "EFFR_ret_1d", "WTI_Oil_FRED_zscore_60d", "EWQ_France_ret_20d", "TED_Spread_vol_20d", "DOW_Price_zscore_60d", "EWM_Malaysia_ret_1d", "vix_mean_abs_ret_5d", "IBEX_Spain_ret_20d"], "is_new": true}, {"model_id": "new_h7_GLOBAL_LogisticRegression_N20_t5", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 7, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "TM_Telephone_vol_20d", "PAYX_Paychex_vol_20d", "CPB_CampbellSoup_zscore_60d", "LMT_LockheedMartin_ret_1d", "NOC_Northrop_ret_20d", "CMCSA_ret_1d", "DHR_vol_20d", "ES_Evergy_ret_1d", "3M_vol_20d", "NWL_Newell_ret_20d", "Nikkei_Japan_vol_20d", "Brent_Oil_FRED_ret_5d", "ITT_ITTInc_ret_5d", "MO_AltriaMG_ret_1d", "MRK_Merck_zscore_60d", "MS_MorganStanley_zscore_60d", "JNJ_ret_1d", "AMD_ret_5d"], "is_new": true}, {"model_id": "new_h7_GLOBAL_LogisticRegression_N20_t6", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 7, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "LUV_SouthwestAir_ret_5d", "Industrial_Production_zscore_60d", "ASX_Australia_vol_20d", "LLY_zscore_60d", "ORCL_vol_20d", "BA_ret_1d", "JNJ_ret_1d", "PG_ret_20d", "DE_Deere_vol_20d", "EWL_Switzerland_vol_20d", "MSTR_Bitcoin3_ret_1d", "heston_var_ev_h3", "HD_ret_20d", "Retail_Sales_zscore_60d", "HangSeng_HK_vol_20d", "US1Y_Rate_ret_5d", "EWS_Singapore_ret_5d", "3M_ret_5d"], "is_new": true}, {"model_id": "new_h7_GLOBAL_LogisticRegression_N20_t7", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 7, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "NWL_Newell_ret_20d", "SBUX_ret_5d", "CCI_CrownCastle_vol_20d", "HangSeng_HK_ret_1d", "ORCL_vol_20d", "EFFR_vol_20d", "EWG_Germany_ret_20d", "ITT_ITTInc_ret_5d", "CPB_CampbellSoup_zscore_60d", "Core_CPI_zscore_60d", "heston_var_ev_h5", "SO_SouthernCo_ret_5d", "spx_abs_ret_max_5d", "ENB_EnbridgeInc_ret_1d", "EOG_EOGResources_ret_5d", "GE_ret_1d", "TXN_vol_20d", "T_ret_1d"], "is_new": true}, {"model_id": "new_h7_GLOBAL_LogisticRegression_N25_t0", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 7, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EQR_Equity_ret_1d", "PAYX_Paychex_ret_20d", "PFE_ret_1d", "CTAS_Cintas_vol_20d", "Brent_Oil_FRED_ret_5d", "EWQ_France_ret_20d", "spx_abs_ret_max_5d", "LLY_zscore_60d", "DAX_Germany_vol_20d", "PAYX_Paychex_vol_20d", "ORCL_vol_20d", "SBUX_ret_5d", "SJM_JM_Smucker_ret_5d", "T_ret_1d", "DOW_Price_zscore_60d", "hmm_p_stress", "LOW_Lowes_ret_20d", "LOW_Lowes_ret_5d", "heston_var_ev_h7", "EWL_Switzerland_zscore_60d", "XLF_Fin_vol_20d", "EWC_Canada_zscore_60d", "EQIX_Equinix_ret_5d"], "is_new": true}, {"model_id": "new_h7_GLOBAL_LogisticRegression_N25_t1", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 7, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "LMT_LockheedMartin_ret_1d", "CCI_CrownCastle_vol_20d", "XLY_Disc_vol_20d", "VRP_ma5", "HD_ret_20d", "AXP_Amex_ret_20d", "EWM_Malaysia_ret_1d", "SJM_JM_Smucker_ret_5d", "BLK_BlackRock_zscore_60d", "XLK_Tech_zscore_60d", "QQQ_vol_20d", "US1Y_Rate_ret_5d", "PG_ret_20d", "SLB_Schlumberger_ret_5d", "INTC_ret_1d", "BTI_BritishAmerican_ret_20d", "HangSeng_HK_ret_1d", "LOW_Lowes_ret_5d", "EWY_Korea_ret_20d", "heston_var_ev_h5", "XOM_ret_20d", "XLV_Health_zscore_60d", "EQIX_Equinix_ret_5d"], "is_new": true}, {"model_id": "new_h7_GLOBAL_LogisticRegression_N25_t2", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 7, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWY_Korea_zscore_60d", "heston_var_ev_h5", "PG_ret_20d", "AMT_AmericanTower_ret_1d", "heston_var_ev_h3", "US7Y_Rate_ret_20d", "SLB_Schlumberger_ret_5d", "NFCI_ret_5d", "EWQ_France_zscore_60d", "NVDA_vol_20d", "spx_vol_5d", "US5Y_Rate_ret_5d", "EXC_Exelon_ret_1d", "EWL_Switzerland_vol_20d", "EWA_Australia_ret_1d", "US3M_Rate_vol_20d", "ITT_ITTInc_ret_5d", "US3M_Rate_zscore_60d", "JNJ_ret_1d", "MSTR_Bitcoin3_ret_20d", "GILD_Gilead_ret_20d", "DOW_Price_zscore_60d", "DHR_vol_20d"], "is_new": true}, {"model_id": "new_h7_GLOBAL_LogisticRegression_N25_t3", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 7, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "US6M_Rate_ret_20d", "AVB_AvalonBay_zscore_60d", "TGT_Target_zscore_60d", "BTI_BritishAmerican_ret_20d", "PG_ret_20d", "EQIX_Equinix_ret_5d", "DHR_ret_1d", "LOW_Lowes_ret_5d", "SPY_zscore_60d", "BLK_BlackRock_zscore_60d", "EWA_Australia_ret_1d", "vix_mean_abs_ret_5d", "SBUX_zscore_60d", "DE_Deere_ret_5d", "Retail_Sales_zscore_60d", "SJM_JM_Smucker_ret_5d", "CTAS_Cintas_vol_20d", "NEE_NextEra_ret_20d", "EQR_Equity_ret_1d", "EWJ_Japan_vol_20d", "XLB_Materials_zscore_60d", "HUM_Humana_ret_5d", "US3Y_Rate_ret_5d"], "is_new": true}, {"model_id": "new_h7_GLOBAL_LogisticRegression_N25_t4", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 7, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "INTC_ret_5d", "SBUX_ret_5d", "SLB_Schlumberger_ret_1d", "EXC_Exelon_zscore_60d", "ASX_Australia_vol_20d", "CI_Cigna_vol_20d", "EWG_Germany_vol_20d", "MSTR_Bitcoin3_ret_5d", "Nikkei_Japan_vol_20d", "DE_Deere_vol_20d", "DOW_Price_zscore_60d", "BDX_Becton_Dickinson_ret_20d", "heston_var_ev_h3", "EWJ_Japan_vol_20d", "EWG_Germany_ret_20d", "M_Macys_vol_20d", "AORD_AUS_zscore_60d", "Michigan_Sentiment_ret_20d", "XLB_Materials_zscore_60d", "spx_abs_ret_max_5d", "EQR_Equity_ret_1d", "QQQ_vol_20d", "JNJ_ret_1d"], "is_new": true}, {"model_id": "new_h7_GLOBAL_LogisticRegression_N25_t5", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 7, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "DHR_vol_20d", "LMT_LockheedMartin_ret_1d", "EWM_Malaysia_zscore_60d", "EWY_Korea_ret_20d", "heston_var_ev_h3", "SJM_JM_Smucker_ret_1d", "US6M_Rate_ret_20d", "US30Y_Rate_ret_20d", "AMZN_ret_5d", "INTC_ret_5d", "spx_momentum_3d", "ASX_Australia_vol_20d", "T_ret_1d", "CLX_Clorox_vol_20d", "ORCL_vol_20d", "WTI_Oil_FRED_zscore_60d", "INTC_ret_1d", "Retail_Sales_zscore_60d", "XOM_ret_1d", "PAYX_Paychex_vol_20d", "DOW_Price_zscore_60d", "EWH_HongKong_ret_5d", "AMT_AmericanTower_ret_1d"], "is_new": true}, {"model_id": "new_h7_GLOBAL_LogisticRegression_N25_t6", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 7, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "spx_momentum_3d", "vix_acceleration_1d", "XLF_Fin_vol_20d", "SJM_JM_Smucker_ret_5d", "EWG_Germany_ret_20d", "AXP_Amex_vol_20d", "LMT_LockheedMartin_ret_1d", "CPB_CampbellSoup_ret_20d", "DE_Deere_vol_20d", "EQR_Equity_ret_1d", "AORD_AUS_zscore_60d", "M_Macys_vol_20d", "EWA_Australia_ret_1d", "TGT_Target_zscore_60d", "ORCL_zscore_60d", "BLK_BlackRock_zscore_60d", "PAYX_Paychex_zscore_60d", "TM_Telephone_ret_1d", "CLX_Clorox_vol_20d", "LLY_zscore_60d", "ES_Evergy_ret_1d", "Brent_Oil_FRED_ret_5d", "TED_Spread_vol_20d"], "is_new": true}, {"model_id": "new_h7_GLOBAL_LogisticRegression_N25_t7", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 7, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "IYR_US_REIT2_zscore_60d", "DE_Deere_vol_20d", "spx_abs_ret_max_5d", "MRK_Merck_zscore_60d", "gjr_condvar_h1", "PAYX_Paychex_vol_20d", "HD_ret_20d", "MS_MorganStanley_zscore_60d", "FedFunds_zscore_60d", "CI_Cigna_vol_20d", "LOW_Lowes_ret_20d", "PLD_Prologis_ret_5d", "EWY_Korea_zscore_60d", "TM_Telephone_vol_20d", "AMD_ret_5d", "SJM_JM_Smucker_ret_5d", "MSTR_Bitcoin3_ret_5d", "XLF_Fin_vol_20d", "EWA_Australia_zscore_60d", "heston_ev_h3", "Nikkei_Japan_zscore_60d", "HD_ret_5d", "TED_Spread_zscore_60d"], "is_new": true}, {"model_id": "new_h7_GLOBAL_LogisticRegression_N30_t0", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 7, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWM_Malaysia_zscore_60d", "NEE_NextEra_ret_20d", "DAX_Germany_vol_20d", "BDX_Becton_Dickinson_ret_20d", "EWY_Korea_ret_20d", "TGT_Target_zscore_60d", "PAYX_Paychex_ret_20d", "HangSeng_HK_ret_5d", "T_ret_1d", "PAYX_Paychex_zscore_60d", "CPB_CampbellSoup_ret_5d", "MS_MorganStanley_ret_1d", "EXC_Exelon_ret_1d", "HangSeng_HK_ret_1d", "LLY_zscore_60d", "M_Macys_vol_20d", "AORD_AUS_zscore_60d", "DHR_vol_20d", "spx_momentum_3d", "AVB_AvalonBay_zscore_60d", "IYM_BasicMaterials_ret_20d", "EWH_HongKong_ret_5d", "EWS_Singapore_ret_5d", "Core_CPI_zscore_60d", "NWL_Newell_ret_20d", "HD_ret_20d", "AMD_ret_1d", "BA_ret_1d"], "is_new": true}, {"model_id": "new_h7_GLOBAL_LogisticRegression_N30_t1", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 7, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "HD_zscore_60d", "US3M_Rate_vol_20d", "PAYX_Paychex_vol_20d", "MS_MorganStanley_ret_5d", "LUV_SouthwestAir_ret_5d", "CCI_CrownCastle_vol_20d", "vix_acceleration_1d", "3M_ret_5d", "CPB_CampbellSoup_ret_5d", "Retail_Sales_zscore_60d", "GILD_Gilead_ret_20d", "US5Y_Rate_ret_5d", "WTI_Oil_FRED_zscore_60d", "EWL_Switzerland_zscore_60d", "SPY_zscore_60d", "EWA_Australia_ret_1d", "PG_ret_20d", "US7Y_Rate_ret_20d", "spx_vol_5d", "SJM_JM_Smucker_ret_5d", "LLY_zscore_60d", "QQQ_vol_20d", "AORD_AUS_zscore_60d", "VVIX_ret_20d", "EXC_Exelon_zscore_60d", "SBUX_ret_5d", "BDX_Becton_Dickinson_ret_20d", "ORCL_zscore_60d"], "is_new": true}, {"model_id": "new_h7_GLOBAL_LogisticRegression_N30_t2", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 7, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "AMZN_ret_5d", "HUM_Humana_ret_5d", "DIS_vol_20d", "LMT_LockheedMartin_vol_20d", "EFFR_ret_1d", "PG_ret_20d", "MSTR_Bitcoin3_ret_20d", "XLY_Disc_vol_20d", "MRK_Merck_zscore_60d", "US7Y_Rate_ret_20d", "ITT_ITTInc_ret_5d", "EWM_Malaysia_zscore_60d", "HD_zscore_60d", "DAX_Germany_vol_20d", "BA_ret_1d", "PLD_Prologis_ret_5d", "HD_ret_20d", "HangSeng_HK_vol_20d", "EWM_Malaysia_ret_1d", "XLV_Health_zscore_60d", "EWA_Australia_zscore_60d", "EOG_EOGResources_vol_20d", "AMT_AmericanTower_ret_1d", "US1Y_Rate_ret_20d", "Core_CPI_zscore_60d", "EXC_Exelon_zscore_60d", "AORD_AUS_zscore_60d", "CI_Cigna_vol_20d"], "is_new": true}, {"model_id": "new_h7_GLOBAL_LogisticRegression_N30_t3", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 7, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "HangSeng_HK_ret_5d", "ES_Evergy_ret_1d", "DHR_vol_20d", "EWG_Germany_ret_20d", "MO_AltriaMG_ret_1d", "VOD_Vodafone_zscore_60d", "Nikkei_Japan_vol_20d", "AMZN_ret_5d", "LLY_zscore_60d", "EWG_Germany_vol_20d", "US3M_Rate_vol_20d", "NVDA_vol_20d", "EFFR_vol_20d", "HangSeng_HK_ret_1d", "SO_SouthernCo_ret_5d", "SBUX_zscore_60d", "ITT_ITTInc_ret_5d", "HD_ret_5d", "XLV_Health_zscore_60d", "ENB_EnbridgeInc_ret_1d", "HD_zscore_60d", "SPY_zscore_60d", "ORCL_zscore_60d", "CTAS_Cintas_vol_20d", "EFFR_ret_1d", "NOC_Northrop_ret_20d", "EWQ_France_zscore_60d", "HD_ret_20d"], "is_new": true}, {"model_id": "new_h7_GLOBAL_LogisticRegression_N30_t4", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 7, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "GD_GeneralDynamics_zscore_60d", "EWJ_Japan_vol_20d", "Retail_Sales_zscore_60d", "CMCSA_ret_1d", "HD_zscore_60d", "AORD_AUS_zscore_60d", "SLB_Schlumberger_ret_5d", "ES_Evergy_ret_1d", "US1Y_Rate_ret_5d", "NEE_NextEra_ret_20d", "M_Macys_vol_20d", "EWS_Singapore_ret_5d", "spx_momentum_3d", "US1Y_Rate_ret_20d", "VRP_ma5", "EMR_Emerson_ret_20d", "US7Y_Rate_ret_20d", "3M_vol_20d", "CPB_CampbellSoup_vol_20d", "ITT_ITTInc_ret_5d", "EWY_Korea_zscore_60d", "HUM_Humana_ret_5d", "hmm_p_stress", "BTI_BritishAmerican_ret_5d", "HangSeng_HK_ret_1d", "AMD_ret_5d", "WTI_Oil_FRED_zscore_60d", "IYM_BasicMaterials_ret_20d"], "is_new": true}, {"model_id": "new_h7_GLOBAL_LogisticRegression_N30_t5", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 7, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "AVB_AvalonBay_zscore_60d", "AXP_Amex_vol_20d", "MS_MorganStanley_zscore_60d", "gjr_condvar_h1", "FedFunds_zscore_60d", "MS_MorganStanley_ret_1d", "EWY_Korea_zscore_60d", "CPB_CampbellSoup_ret_5d", "SJM_JM_Smucker_ret_1d", "EWY_Korea_ret_20d", "CPB_CampbellSoup_ret_20d", "EWG_Germany_ret_20d", "JNJ_ret_1d", "TM_Telephone_vol_20d", "ASX_Australia_ret_5d", "WTI_Oil_FRED_zscore_60d", "HD_ret_5d", "XOM_ret_1d", "hmm_p_stress", "PPL_PPL_ret_1d", "EMR_Emerson_ret_20d", "MSTR_Bitcoin3_ret_20d", "TED_Spread_zscore_60d", "CI_Cigna_vol_20d", "BTI_BritishAmerican_ret_20d", "heston_var_ev_h5", "AMZN_ret_5d", "HD_zscore_60d"], "is_new": true}, {"model_id": "new_h7_GLOBAL_LogisticRegression_N30_t6", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 7, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "PAYX_Paychex_zscore_60d", "HangSeng_HK_vol_20d", "VOD_Vodafone_zscore_60d", "BLK_BlackRock_zscore_60d", "NFCI_ret_5d", "FedFunds_zscore_60d", "ORCL_vol_20d", "EWG_Germany_ret_20d", "LMT_LockheedMartin_vol_20d", "CI_Cigna_vol_20d", "heston_ev_h3", "HD_ret_5d", "AMD_ret_1d", "CPB_CampbellSoup_ret_20d", "WTI_Oil_FRED_zscore_60d", "BA_ret_1d", "T10Y2Y_Spread_ret_5d", "HangSeng_HK_ret_5d", "US30Y_Rate_ret_20d", "heston_var_ev_h3", "EWJ_Japan_vol_20d", "EWS_Singapore_ret_5d", "BTI_BritishAmerican_ret_20d", "EWL_Switzerland_vol_20d", "XLV_Health_zscore_60d", "INTC_ret_1d", "MRK_Merck_zscore_60d", "ORCL_zscore_60d"], "is_new": true}, {"model_id": "new_h7_GLOBAL_LogisticRegression_N30_t7", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 7, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "NFCI_ret_5d", "Industrial_Production_zscore_60d", "IWM_SmallCap_vol_20d", "EWL_Switzerland_zscore_60d", "DHR_ret_1d", "EWS_Singapore_ret_5d", "DOW_Price_zscore_60d", "PAYX_Paychex_ret_20d", "EWM_Malaysia_vol_20d", "EWA_Australia_zscore_60d", "PFE_ret_1d", "WTI_Oil_FRED_zscore_60d", "Core_PCE_zscore_60d", "TED_Spread_zscore_60d", "AMD_ret_1d", "EQR_Equity_ret_1d", "JNJ_ret_1d", "AXP_Amex_ret_20d", "DE_Deere_ret_5d", "EFFR_ret_1d", "Nikkei_Japan_vol_20d", "MS_MorganStanley_zscore_60d", "LUV_SouthwestAir_ret_5d", "DHR_vol_20d", "TM_Telephone_vol_20d", "EOG_EOGResources_ret_5d", "Michigan_Sentiment_ret_20d", "HD_ret_5d"], "is_new": true}, {"model_id": "new_h10_CALM_XGBoost_N5_t0", "algo": "XGBoost", "regime": "CALM", "horizon": 10, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EFFR_ret_1d", "DIS_vol_20d", "EXC_Exelon_ret_1d"], "is_new": true}, {"model_id": "new_h10_CALM_XGBoost_N5_t1", "algo": "XGBoost", "regime": "CALM", "horizon": 10, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "SLB_Schlumberger_ret_1d", "INTC_ret_1d", "Nikkei_Japan_zscore_60d"], "is_new": true}, {"model_id": "new_h10_CALM_XGBoost_N5_t2", "algo": "XGBoost", "regime": "CALM", "horizon": 10, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWA_Australia_ret_1d", "DE_Deere_ret_5d", "EOG_EOGResources_vol_20d"], "is_new": true}, {"model_id": "new_h10_CALM_XGBoost_N5_t3", "algo": "XGBoost", "regime": "CALM", "horizon": 10, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "AVB_AvalonBay_zscore_60d", "TM_Telephone_vol_20d", "EWY_Korea_zscore_60d"], "is_new": true}, {"model_id": "new_h10_CALM_XGBoost_N5_t4", "algo": "XGBoost", "regime": "CALM", "horizon": 10, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "US30Y_Rate_ret_20d", "US7Y_Rate_ret_20d", "EFFR_ret_1d"], "is_new": true}, {"model_id": "new_h10_CALM_XGBoost_N5_t5", "algo": "XGBoost", "regime": "CALM", "horizon": 10, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "XLF_Fin_vol_20d", "BLK_BlackRock_zscore_60d", "ORCL_vol_20d"], "is_new": true}, {"model_id": "new_h10_CALM_XGBoost_N5_t6", "algo": "XGBoost", "regime": "CALM", "horizon": 10, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EOG_EOGResources_vol_20d", "Brent_Oil_FRED_ret_20d", "EXC_Exelon_zscore_60d"], "is_new": true}, {"model_id": "new_h10_CALM_XGBoost_N5_t7", "algo": "XGBoost", "regime": "CALM", "horizon": 10, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "PG_ret_20d", "NVDA_vol_20d", "EXC_Exelon_zscore_60d"], "is_new": true}, {"model_id": "new_h10_CALM_XGBoost_N8_t0", "algo": "XGBoost", "regime": "CALM", "horizon": 10, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "GE_ret_1d", "CPB_CampbellSoup_ret_20d", "BLK_BlackRock_zscore_60d", "SLB_Schlumberger_ret_1d", "XOM_ret_20d", "TED_Spread_vol_20d"], "is_new": true}, {"model_id": "new_h10_CALM_XGBoost_N8_t1", "algo": "XGBoost", "regime": "CALM", "horizon": 10, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "Michigan_Sentiment_ret_20d", "CPB_CampbellSoup_ret_5d", "US1Y_Rate_ret_20d", "SLB_Schlumberger_ret_5d", "IBEX_Spain_ret_20d", "LMT_LockheedMartin_ret_1d"], "is_new": true}, {"model_id": "new_h10_CALM_XGBoost_N8_t2", "algo": "XGBoost", "regime": "CALM", "horizon": 10, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "INTC_ret_1d", "DHR_vol_20d", "HangSeng_HK_ret_1d", "EWM_Malaysia_vol_20d", "M_Macys_vol_20d", "SLB_Schlumberger_ret_5d"], "is_new": true}, {"model_id": "new_h10_CALM_XGBoost_N8_t3", "algo": "XGBoost", "regime": "CALM", "horizon": 10, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "US1Y_Rate_ret_20d", "EWG_Germany_vol_20d", "TM_Telephone_vol_20d", "PG_ret_20d", "US6M_Rate_ret_20d", "PPL_PPL_ret_1d"], "is_new": true}, {"model_id": "new_h10_CALM_XGBoost_N8_t4", "algo": "XGBoost", "regime": "CALM", "horizon": 10, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "TXN_vol_20d", "IYR_US_REIT2_zscore_60d", "EWA_Australia_zscore_60d", "EQR_Equity_ret_1d", "Core_CPI_zscore_60d", "QQQ_vol_20d"], "is_new": true}, {"model_id": "new_h10_CALM_XGBoost_N8_t5", "algo": "XGBoost", "regime": "CALM", "horizon": 10, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "HD_ret_5d", "Michigan_Sentiment_ret_20d", "XLV_Health_zscore_60d", "MO_AltriaMG_ret_1d", "GILD_Gilead_ret_20d", "MSTR_Bitcoin3_ret_20d"], "is_new": true}, {"model_id": "new_h10_CALM_XGBoost_N8_t6", "algo": "XGBoost", "regime": "CALM", "horizon": 10, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWC_Canada_zscore_60d", "CPB_CampbellSoup_vol_20d", "heston_var_ev_h3", "HD_zscore_60d", "LUV_SouthwestAir_ret_5d", "PAYX_Paychex_vol_20d"], "is_new": true}, {"model_id": "new_h10_CALM_XGBoost_N8_t7", "algo": "XGBoost", "regime": "CALM", "horizon": 10, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "DOW_Price_zscore_60d", "ES_Evergy_ret_1d", "LMT_LockheedMartin_vol_20d", "CI_Cigna_vol_20d", "HUM_Humana_ret_5d", "Michigan_Sentiment_ret_20d"], "is_new": true}, {"model_id": "new_h10_CALM_XGBoost_N10_t0", "algo": "XGBoost", "regime": "CALM", "horizon": 10, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "ENB_EnbridgeInc_ret_1d", "NWL_Newell_ret_20d", "XLB_Materials_zscore_60d", "PAYX_Paychex_zscore_60d", "heston_var_ev_h5", "BTI_BritishAmerican_ret_20d", "SBUX_ret_5d", "spx_momentum_3d"], "is_new": true}, {"model_id": "new_h10_CALM_XGBoost_N10_t1", "algo": "XGBoost", "regime": "CALM", "horizon": 10, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "SO_SouthernCo_ret_5d", "Michigan_Sentiment_ret_20d", "GE_ret_1d", "EQR_Equity_ret_1d", "SBUX_vol_20d", "BTI_BritishAmerican_ret_5d", "hmm_p_stress", "IWM_SmallCap_vol_20d"], "is_new": true}, {"model_id": "new_h10_CALM_XGBoost_N10_t2", "algo": "XGBoost", "regime": "CALM", "horizon": 10, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "hmm_p_stress", "EWQ_France_zscore_60d", "XLB_Materials_zscore_60d", "HUM_Humana_ret_5d", "EWL_Switzerland_zscore_60d", "MS_MorganStanley_ret_5d", "Michigan_Sentiment_ret_20d", "INTC_ret_5d"], "is_new": true}, {"model_id": "new_h10_CALM_XGBoost_N10_t3", "algo": "XGBoost", "regime": "CALM", "horizon": 10, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "PPL_PPL_ret_1d", "HangSeng_HK_vol_20d", "PAYX_Paychex_vol_20d", "EWG_Germany_vol_20d", "BTI_BritishAmerican_ret_5d", "XLB_Materials_zscore_60d", "TED_Spread_zscore_60d", "SBUX_ret_5d"], "is_new": true}, {"model_id": "new_h10_CALM_XGBoost_N10_t4", "algo": "XGBoost", "regime": "CALM", "horizon": 10, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "CPB_CampbellSoup_vol_20d", "LMT_LockheedMartin_ret_1d", "DE_Deere_vol_20d", "SJM_JM_Smucker_ret_1d", "EWS_Singapore_ret_5d", "TM_Telephone_vol_20d", "CLX_Clorox_vol_20d", "LOW_Lowes_ret_5d"], "is_new": true}, {"model_id": "new_h10_CALM_XGBoost_N10_t5", "algo": "XGBoost", "regime": "CALM", "horizon": 10, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "MO_AltriaMG_ret_1d", "PAYX_Paychex_zscore_60d", "CPB_CampbellSoup_ret_5d", "SJM_JM_Smucker_ret_1d", "LOW_Lowes_ret_5d", "EOG_EOGResources_ret_5d", "M_Macys_vol_20d", "CMCSA_ret_1d"], "is_new": true}, {"model_id": "new_h10_CALM_XGBoost_N10_t6", "algo": "XGBoost", "regime": "CALM", "horizon": 10, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "ORCL_zscore_60d", "WTI_Oil_FRED_zscore_60d", "SJM_JM_Smucker_ret_5d", "EOG_EOGResources_vol_20d", "SBUX_zscore_60d", "EWL_Switzerland_zscore_60d", "ASX_Australia_ret_5d", "PG_ret_20d"], "is_new": true}, {"model_id": "new_h10_CALM_XGBoost_N10_t7", "algo": "XGBoost", "regime": "CALM", "horizon": 10, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "AMGN_Amgen_ret_1d", "LOW_Lowes_ret_20d", "EWG_Germany_ret_20d", "EWY_Korea_zscore_60d", "EOG_EOGResources_vol_20d", "3M_vol_20d", "EFFR_ret_1d", "Nikkei_Japan_vol_20d"], "is_new": true}, {"model_id": "new_h10_CALM_XGBoost_N12_t0", "algo": "XGBoost", "regime": "CALM", "horizon": 10, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "SBUX_zscore_60d", "AMT_AmericanTower_ret_1d", "SPY_zscore_60d", "heston_var_ev_h7", "IYM_BasicMaterials_ret_20d", "CPB_CampbellSoup_ret_5d", "VOD_Vodafone_zscore_60d", "HangSeng_HK_ret_1d", "heston_var_ev_h3", "CCI_CrownCastle_vol_20d"], "is_new": true}, {"model_id": "new_h10_CALM_XGBoost_N12_t1", "algo": "XGBoost", "regime": "CALM", "horizon": 10, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "DAX_Germany_vol_20d", "INTC_ret_1d", "ITT_ITTInc_ret_5d", "EWH_HongKong_ret_5d", "US1Y_Rate_ret_20d", "EWA_Australia_ret_1d", "CTAS_Cintas_vol_20d", "MSTR_Bitcoin3_ret_1d", "Core_PCE_zscore_60d", "LMT_LockheedMartin_vol_20d"], "is_new": true}, {"model_id": "new_h10_CALM_XGBoost_N12_t2", "algo": "XGBoost", "regime": "CALM", "horizon": 10, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EMR_Emerson_ret_20d", "EWH_HongKong_ret_5d", "ORCL_zscore_60d", "Retail_Sales_zscore_60d", "XLB_Materials_zscore_60d", "NEE_NextEra_ret_20d", "HangSeng_HK_vol_20d", "GD_GeneralDynamics_zscore_60d", "EWG_Germany_ret_20d", "XOM_ret_1d"], "is_new": true}, {"model_id": "new_h10_CALM_XGBoost_N12_t3", "algo": "XGBoost", "regime": "CALM", "horizon": 10, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "VOD_Vodafone_zscore_60d", "heston_var_ev_h7", "DAX_Germany_zscore_60d", "EWM_Malaysia_vol_20d", "PAYX_Paychex_zscore_60d", "CTAS_Cintas_vol_20d", "SBUX_vol_20d", "AXP_Amex_vol_20d", "EWL_Switzerland_zscore_60d", "LUV_SouthwestAir_ret_5d"], "is_new": true}, {"model_id": "new_h10_CALM_XGBoost_N12_t4", "algo": "XGBoost", "regime": "CALM", "horizon": 10, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "US6M_Rate_ret_20d", "XOM_ret_1d", "VRP_ma5", "EWQ_France_ret_20d", "DAX_Germany_vol_20d", "XLY_Disc_vol_20d", "MS_MorganStanley_zscore_60d", "EXC_Exelon_ret_1d", "BLK_BlackRock_zscore_60d", "Retail_Sales_zscore_60d"], "is_new": true}, {"model_id": "new_h10_CALM_XGBoost_N12_t5", "algo": "XGBoost", "regime": "CALM", "horizon": 10, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "AVB_AvalonBay_zscore_60d", "AMT_AmericanTower_ret_1d", "HangSeng_HK_vol_20d", "spx_abs_ret_max_5d", "EWL_Switzerland_zscore_60d", "TED_Spread_vol_20d", "LOW_Lowes_ret_5d", "EQR_Equity_ret_1d", "SPY_zscore_60d", "NVDA_vol_20d"], "is_new": true}, {"model_id": "new_h10_CALM_XGBoost_N12_t6", "algo": "XGBoost", "regime": "CALM", "horizon": 10, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWH_HongKong_ret_5d", "DE_Deere_vol_20d", "SBUX_ret_5d", "AMZN_ret_5d", "HangSeng_HK_ret_5d", "EWS_Singapore_ret_5d", "WTI_Oil_FRED_zscore_60d", "CPB_CampbellSoup_vol_20d", "ORCL_zscore_60d", "US7Y_Rate_ret_20d"], "is_new": true}, {"model_id": "new_h10_CALM_XGBoost_N12_t7", "algo": "XGBoost", "regime": "CALM", "horizon": 10, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "DE_Deere_ret_5d", "EWH_HongKong_ret_5d", "CPB_CampbellSoup_vol_20d", "DAX_Germany_vol_20d", "AXP_Amex_vol_20d", "XOM_ret_1d", "EWG_Germany_vol_20d", "SLB_Schlumberger_ret_5d", "PPL_PPL_ret_1d", "TGT_Target_zscore_60d"], "is_new": true}, {"model_id": "new_h10_CALM_XGBoost_N15_t0", "algo": "XGBoost", "regime": "CALM", "horizon": 10, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "MS_MorganStanley_ret_1d", "INTC_ret_1d", "EWG_Germany_ret_20d", "LUV_SouthwestAir_ret_5d", "US7Y_Rate_ret_20d", "SBUX_vol_20d", "DOW_Price_zscore_60d", "US6M_Rate_ret_20d", "EWJ_Japan_vol_20d", "Brent_Oil_FRED_ret_20d", "EQR_Equity_ret_1d", "spx_momentum_3d", "EWA_Australia_ret_1d"], "is_new": true}, {"model_id": "new_h10_CALM_XGBoost_N15_t1", "algo": "XGBoost", "regime": "CALM", "horizon": 10, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "Nikkei_Japan_zscore_60d", "US7Y_Rate_ret_20d", "SPY_zscore_60d", "TXN_vol_20d", "M_Macys_vol_20d", "HD_ret_20d", "SBUX_vol_20d", "Brent_Oil_FRED_ret_20d", "heston_ev_h3", "heston_var_ev_h7", "VVIX_ret_20d", "EXC_Exelon_ret_1d", "XLB_Materials_zscore_60d"], "is_new": true}, {"model_id": "new_h10_CALM_XGBoost_N15_t2", "algo": "XGBoost", "regime": "CALM", "horizon": 10, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWY_Korea_ret_20d", "PFE_ret_1d", "MS_MorganStanley_zscore_60d", "AMZN_ret_5d", "CMCSA_ret_1d", "SJM_JM_Smucker_ret_1d", "XLV_Health_zscore_60d", "HangSeng_HK_vol_20d", "EWS_Singapore_ret_5d", "FedFunds_zscore_60d", "EWM_Malaysia_ret_1d", "EWH_HongKong_ret_5d", "SCHW_Schwab_ret_5d"], "is_new": true}, {"model_id": "new_h10_CALM_XGBoost_N15_t3", "algo": "XGBoost", "regime": "CALM", "horizon": 10, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "PAYX_Paychex_vol_20d", "DHR_vol_20d", "VVIX_ret_20d", "EWH_HongKong_ret_5d", "CPB_CampbellSoup_zscore_60d", "DIS_vol_20d", "EFFR_ret_1d", "SJM_JM_Smucker_ret_5d", "MSTR_Bitcoin3_ret_5d", "US1Y_Rate_ret_5d", "EWJ_Japan_vol_20d", "INTC_ret_1d", "CMCSA_ret_1d"], "is_new": true}, {"model_id": "new_h10_CALM_XGBoost_N15_t4", "algo": "XGBoost", "regime": "CALM", "horizon": 10, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "MO_AltriaMG_ret_1d", "NVDA_vol_20d", "EWS_Singapore_ret_5d", "HangSeng_HK_ret_5d", "EWA_Australia_zscore_60d", "BTI_BritishAmerican_ret_20d", "ES_Evergy_ret_1d", "Core_PCE_zscore_60d", "NWL_Newell_ret_20d", "US7Y_Rate_ret_20d", "AMZN_ret_5d", "VOD_Vodafone_zscore_60d", "BLK_BlackRock_zscore_60d"], "is_new": true}, {"model_id": "new_h10_CALM_XGBoost_N15_t5", "algo": "XGBoost", "regime": "CALM", "horizon": 10, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "MS_MorganStanley_zscore_60d", "heston_ev_h3", "TXN_vol_20d", "spx_vol_5d", "EFFR_ret_1d", "T_ret_1d", "PFE_ret_1d", "AMT_AmericanTower_ret_1d", "AMGN_Amgen_ret_1d", "SBUX_ret_5d", "MS_MorganStanley_ret_5d", "EOG_EOGResources_vol_20d", "IYR_US_REIT2_zscore_60d"], "is_new": true}, {"model_id": "new_h10_CALM_XGBoost_N15_t6", "algo": "XGBoost", "regime": "CALM", "horizon": 10, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "HD_ret_1d", "WTI_Oil_FRED_zscore_60d", "EQR_Equity_ret_1d", "MSTR_Bitcoin3_ret_5d", "CCI_CrownCastle_vol_20d", "HD_ret_20d", "EMR_Emerson_ret_20d", "LOW_Lowes_ret_20d", "BTI_BritishAmerican_ret_20d", "XLB_Materials_zscore_60d", "HD_ret_5d", "hmm_p_stress", "SLB_Schlumberger_ret_5d"], "is_new": true}, {"model_id": "new_h10_CALM_XGBoost_N15_t7", "algo": "XGBoost", "regime": "CALM", "horizon": 10, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "TM_Telephone_vol_20d", "EWJ_Japan_vol_20d", "MS_MorganStanley_zscore_60d", "EWG_Germany_vol_20d", "EWG_Germany_ret_20d", "EWY_Korea_ret_20d", "CI_Cigna_vol_20d", "VVIX_ret_20d", "EWQ_France_zscore_60d", "EOG_EOGResources_vol_20d", "vix_acceleration_1d", "HangSeng_HK_ret_1d", "MSTR_Bitcoin3_ret_20d"], "is_new": true}, {"model_id": "new_h10_CALM_XGBoost_N20_t0", "algo": "XGBoost", "regime": "CALM", "horizon": 10, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWS_Singapore_ret_5d", "DE_Deere_vol_20d", "US1Y_Rate_ret_20d", "EMR_Emerson_ret_20d", "BLK_BlackRock_zscore_60d", "spx_vol_5d", "Core_CPI_zscore_60d", "INTC_ret_5d", "LMT_LockheedMartin_ret_1d", "AXP_Amex_vol_20d", "ORCL_vol_20d", "SJM_JM_Smucker_ret_1d", "LMT_LockheedMartin_vol_20d", "hmm_p_stress", "ENB_EnbridgeInc_ret_1d", "AMGN_Amgen_ret_1d", "Core_PCE_zscore_60d", "TED_Spread_vol_20d"], "is_new": true}, {"model_id": "new_h10_CALM_XGBoost_N20_t1", "algo": "XGBoost", "regime": "CALM", "horizon": 10, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "Core_CPI_zscore_60d", "HangSeng_HK_vol_20d", "M_Macys_vol_20d", "INTC_ret_5d", "EXC_Exelon_ret_1d", "HD_ret_20d", "XLF_Fin_vol_20d", "WTI_Oil_FRED_zscore_60d", "TED_Spread_zscore_60d", "MS_MorganStanley_ret_5d", "VRP_ma5", "ASX_Australia_ret_5d", "JNJ_ret_1d", "ENB_EnbridgeInc_ret_1d", "QQQ_vol_20d", "EMR_Emerson_ret_20d", "BDX_Becton_Dickinson_ret_20d", "HangSeng_HK_ret_5d"], "is_new": true}, {"model_id": "new_h10_CALM_XGBoost_N20_t2", "algo": "XGBoost", "regime": "CALM", "horizon": 10, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "DOW_Price_zscore_60d", "BTI_BritishAmerican_ret_20d", "HD_ret_1d", "LUV_SouthwestAir_ret_5d", "IYR_US_REIT2_zscore_60d", "NVDA_vol_20d", "NOC_Northrop_ret_20d", "HD_zscore_60d", "GILD_Gilead_ret_20d", "LOW_Lowes_ret_5d", "QQQ_vol_20d", "vix_mean_abs_ret_5d", "PFE_ret_1d", "AXP_Amex_vol_20d", "3M_ret_5d", "US1Y_Rate_ret_5d", "AMT_AmericanTower_ret_1d", "MRK_Merck_zscore_60d"], "is_new": true}, {"model_id": "new_h10_CALM_XGBoost_N20_t3", "algo": "XGBoost", "regime": "CALM", "horizon": 10, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "TM_Telephone_ret_1d", "XOM_ret_1d", "LOW_Lowes_ret_20d", "GILD_Gilead_ret_20d", "EFFR_ret_1d", "SJM_JM_Smucker_ret_5d", "IYR_US_REIT2_zscore_60d", "EQR_Equity_ret_1d", "EWJ_Japan_vol_20d", "CMCSA_ret_1d", "IWM_SmallCap_vol_20d", "VRP_ma5", "XLF_Fin_vol_20d", "T10Y2Y_Spread_ret_5d", "EWA_Australia_zscore_60d", "DE_Deere_ret_5d", "DHR_vol_20d", "HangSeng_HK_ret_1d"], "is_new": true}, {"model_id": "new_h10_CALM_XGBoost_N20_t4", "algo": "XGBoost", "regime": "CALM", "horizon": 10, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "TM_Telephone_ret_1d", "MS_MorganStanley_ret_1d", "NVDA_vol_20d", "MS_MorganStanley_zscore_60d", "BTI_BritishAmerican_ret_20d", "NOC_Northrop_ret_20d", "AORD_AUS_zscore_60d", "Brent_Oil_FRED_ret_5d", "ASX_Australia_ret_5d", "PLD_Prologis_ret_5d", "XLY_Disc_vol_20d", "IBEX_Spain_ret_20d", "LLY_zscore_60d", "AMD_ret_1d", "XLV_Health_zscore_60d", "heston_var_ev_h7", "TXN_vol_20d", "LMT_LockheedMartin_vol_20d"], "is_new": true}, {"model_id": "new_h10_CALM_XGBoost_N20_t5", "algo": "XGBoost", "regime": "CALM", "horizon": 10, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "XLV_Health_zscore_60d", "LOW_Lowes_ret_20d", "Michigan_Sentiment_ret_20d", "BTI_BritishAmerican_ret_20d", "US3M_Rate_zscore_60d", "ITT_ITTInc_ret_5d", "PPL_PPL_ret_1d", "ES_Evergy_ret_1d", "CPB_CampbellSoup_zscore_60d", "SBUX_ret_5d", "SPY_zscore_60d", "XLF_Fin_vol_20d", "GD_GeneralDynamics_zscore_60d", "3M_vol_20d", "EFFR_ret_1d", "MSTR_Bitcoin3_ret_1d", "M_Macys_vol_20d", "spx_vol_5d"], "is_new": true}, {"model_id": "new_h10_CALM_XGBoost_N20_t6", "algo": "XGBoost", "regime": "CALM", "horizon": 10, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWM_Malaysia_ret_1d", "TED_Spread_vol_20d", "SLB_Schlumberger_ret_1d", "IWM_SmallCap_vol_20d", "EWA_Australia_ret_1d", "ASX_Australia_vol_20d", "ITT_ITTInc_ret_5d", "EWG_Germany_ret_20d", "FedFunds_zscore_60d", "EWJ_Japan_vol_20d", "EFFR_ret_1d", "MSTR_Bitcoin3_ret_1d", "Industrial_Production_zscore_60d", "heston_var_ev_h7", "US1Y_Rate_ret_20d", "ES_Evergy_ret_1d", "QQQ_vol_20d", "NEE_NextEra_ret_20d"], "is_new": true}, {"model_id": "new_h10_CALM_XGBoost_N20_t7", "algo": "XGBoost", "regime": "CALM", "horizon": 10, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "gjr_condvar_h1", "TGT_Target_zscore_60d", "SJM_JM_Smucker_ret_5d", "LMT_LockheedMartin_vol_20d", "AXP_Amex_vol_20d", "ENB_EnbridgeInc_ret_1d", "WTI_Oil_FRED_zscore_60d", "XLY_Disc_vol_20d", "BA_ret_1d", "SBUX_vol_20d", "PAYX_Paychex_vol_20d", "NOC_Northrop_ret_20d", "TM_Telephone_vol_20d", "DHR_ret_1d", "heston_var_ev_h5", "PFE_ret_1d", "DIS_vol_20d", "SJM_JM_Smucker_ret_1d"], "is_new": true}, {"model_id": "new_h10_CALM_XGBoost_N25_t0", "algo": "XGBoost", "regime": "CALM", "horizon": 10, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "HangSeng_HK_ret_1d", "DHR_vol_20d", "SPY_zscore_60d", "ASX_Australia_vol_20d", "Industrial_Production_zscore_60d", "T10Y2Y_Spread_ret_5d", "TED_Spread_vol_20d", "ORCL_vol_20d", "US3M_Rate_zscore_60d", "CPB_CampbellSoup_zscore_60d", "IYR_US_REIT2_zscore_60d", "ASX_Australia_ret_5d", "EWA_Australia_zscore_60d", "3M_ret_5d", "heston_var_ev_h3", "CLX_Clorox_vol_20d", "EWQ_France_ret_20d", "PPL_PPL_ret_1d", "CI_Cigna_vol_20d", "ENB_EnbridgeInc_ret_1d", "TGT_Target_zscore_60d", "EWQ_France_zscore_60d", "SCHW_Schwab_ret_5d"], "is_new": true}, {"model_id": "new_h10_CALM_XGBoost_N25_t1", "algo": "XGBoost", "regime": "CALM", "horizon": 10, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "DOW_Price_zscore_60d", "SBUX_zscore_60d", "EWA_Australia_ret_1d", "Nikkei_Japan_vol_20d", "EWM_Malaysia_ret_1d", "PFE_ret_1d", "heston_ev_h3", "VRP_ma5", "EWL_Switzerland_zscore_60d", "JNJ_ret_1d", "TXN_vol_20d", "US6M_Rate_ret_20d", "HD_ret_20d", "EWY_Korea_zscore_60d", "CMCSA_ret_1d", "EXC_Exelon_zscore_60d", "EWQ_France_zscore_60d", "PG_ret_20d", "EFFR_ret_1d", "Industrial_Production_zscore_60d", "SLB_Schlumberger_ret_1d", "XLY_Disc_vol_20d", "XOM_ret_20d"], "is_new": true}, {"model_id": "new_h10_CALM_XGBoost_N25_t2", "algo": "XGBoost", "regime": "CALM", "horizon": 10, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "SLB_Schlumberger_ret_5d", "US3M_Rate_vol_20d", "DHR_vol_20d", "TXN_vol_20d", "IYR_US_REIT2_zscore_60d", "ASX_Australia_ret_5d", "EOG_EOGResources_vol_20d", "Retail_Sales_zscore_60d", "GILD_Gilead_ret_20d", "VVIX_ret_20d", "FedFunds_zscore_60d", "BA_ret_1d", "NFCI_ret_5d", "hmm_p_stress", "MS_MorganStanley_ret_5d", "PAYX_Paychex_vol_20d", "T10Y2Y_Spread_ret_5d", "EWQ_France_ret_20d", "PG_ret_20d", "AXP_Amex_ret_20d", "EWM_Malaysia_vol_20d", "EWC_Canada_zscore_60d", "MSTR_Bitcoin3_ret_5d"], "is_new": true}, {"model_id": "new_h10_CALM_XGBoost_N25_t3", "algo": "XGBoost", "regime": "CALM", "horizon": 10, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "CPB_CampbellSoup_ret_5d", "EOG_EOGResources_vol_20d", "DIS_vol_20d", "EWA_Australia_zscore_60d", "PFE_ret_1d", "EWM_Malaysia_ret_1d", "US3Y_Rate_ret_5d", "DHR_ret_1d", "EFFR_vol_20d", "ASX_Australia_vol_20d", "EWH_HongKong_ret_5d", "BDX_Becton_Dickinson_ret_20d", "PLD_Prologis_ret_5d", "MRK_Merck_zscore_60d", "AMT_AmericanTower_ret_1d", "LOW_Lowes_ret_5d", "QQQ_vol_20d", "XOM_ret_20d", "DE_Deere_vol_20d", "NWL_Newell_ret_20d", "Nikkei_Japan_vol_20d", "AVB_AvalonBay_zscore_60d", "PPL_PPL_ret_1d"], "is_new": true}, {"model_id": "new_h10_CALM_XGBoost_N25_t4", "algo": "XGBoost", "regime": "CALM", "horizon": 10, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "PFE_ret_1d", "US1Y_Rate_ret_20d", "CI_Cigna_vol_20d", "EMR_Emerson_ret_20d", "CCI_CrownCastle_vol_20d", "M_Macys_vol_20d", "AMZN_ret_5d", "HangSeng_HK_vol_20d", "US7Y_Rate_ret_20d", "ASX_Australia_vol_20d", "DAX_Germany_vol_20d", "US30Y_Rate_ret_20d", "VOD_Vodafone_zscore_60d", "spx_momentum_3d", "NVDA_vol_20d", "Industrial_Production_zscore_60d", "BA_ret_1d", "heston_ev_h3", "EWM_Malaysia_vol_20d", "US3M_Rate_vol_20d", "BTI_BritishAmerican_ret_20d", "LOW_Lowes_ret_5d", "MSTR_Bitcoin3_ret_20d"], "is_new": true}, {"model_id": "new_h10_CALM_XGBoost_N25_t5", "algo": "XGBoost", "regime": "CALM", "horizon": 10, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "HD_ret_1d", "PAYX_Paychex_zscore_60d", "CPB_CampbellSoup_ret_5d", "Core_PCE_zscore_60d", "EWH_HongKong_ret_5d", "INTC_ret_1d", "CPB_CampbellSoup_zscore_60d", "EWA_Australia_zscore_60d", "VRP_ma5", "LMT_LockheedMartin_vol_20d", "XLF_Fin_vol_20d", "DIS_vol_20d", "SCHW_Schwab_ret_5d", "EXC_Exelon_zscore_60d", "MSTR_Bitcoin3_ret_5d", "T_ret_1d", "CPB_CampbellSoup_ret_20d", "NWL_Newell_ret_20d", "DOW_Price_zscore_60d", "3M_vol_20d", "EFFR_ret_1d", "ORCL_vol_20d", "SPY_zscore_60d"], "is_new": true}, {"model_id": "new_h10_CALM_XGBoost_N25_t6", "algo": "XGBoost", "regime": "CALM", "horizon": 10, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "LMT_LockheedMartin_ret_1d", "SBUX_ret_5d", "CMCSA_ret_1d", "GD_GeneralDynamics_zscore_60d", "AMT_AmericanTower_ret_1d", "Michigan_Sentiment_ret_20d", "XLB_Materials_zscore_60d", "EOG_EOGResources_ret_5d", "HangSeng_HK_ret_1d", "SCHW_Schwab_ret_5d", "Core_PCE_zscore_60d", "CTAS_Cintas_vol_20d", "DHR_vol_20d", "EWY_Korea_ret_20d", "Industrial_Production_zscore_60d", "CPB_CampbellSoup_ret_20d", "PG_ret_20d", "EWM_Malaysia_vol_20d", "XLV_Health_zscore_60d", "US3Y_Rate_ret_5d", "IYM_BasicMaterials_ret_20d", "SBUX_zscore_60d", "EFFR_ret_1d"], "is_new": true}, {"model_id": "new_h10_CALM_XGBoost_N25_t7", "algo": "XGBoost", "regime": "CALM", "horizon": 10, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "MS_MorganStanley_ret_1d", "heston_var_ev_h7", "EWA_Australia_ret_1d", "BTI_BritishAmerican_ret_5d", "EWG_Germany_ret_20d", "Brent_Oil_FRED_ret_5d", "BDX_Becton_Dickinson_ret_20d", "TM_Telephone_vol_20d", "MO_AltriaMG_ret_1d", "LMT_LockheedMartin_ret_1d", "Nikkei_Japan_zscore_60d", "AXP_Amex_ret_20d", "IBEX_Spain_ret_20d", "US7Y_Rate_ret_20d", "XLY_Disc_vol_20d", "CI_Cigna_vol_20d", "SCHW_Schwab_ret_5d", "MS_MorganStanley_zscore_60d", "LLY_zscore_60d", "HUM_Humana_ret_5d", "HangSeng_HK_ret_5d", "spx_momentum_3d", "PAYX_Paychex_zscore_60d"], "is_new": true}, {"model_id": "new_h10_CALM_XGBoost_N30_t0", "algo": "XGBoost", "regime": "CALM", "horizon": 10, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWM_Malaysia_zscore_60d", "CPB_CampbellSoup_ret_5d", "Nikkei_Japan_vol_20d", "IYR_US_REIT2_zscore_60d", "TGT_Target_zscore_60d", "NVDA_vol_20d", "GE_ret_1d", "PFE_ret_1d", "AXP_Amex_ret_20d", "EQIX_Equinix_ret_5d", "spx_vol_5d", "AORD_AUS_zscore_60d", "CI_Cigna_vol_20d", "Nikkei_Japan_zscore_60d", "T_ret_1d", "heston_var_ev_h3", "EWY_Korea_zscore_60d", "QQQ_vol_20d", "GD_GeneralDynamics_zscore_60d", "EWC_Canada_zscore_60d", "NFCI_ret_5d", "US7Y_Rate_ret_20d", "HD_ret_5d", "MSTR_Bitcoin3_ret_20d", "US30Y_Rate_ret_20d", "BLK_BlackRock_zscore_60d", "LOW_Lowes_ret_5d", "VOD_Vodafone_zscore_60d"], "is_new": true}, {"model_id": "new_h10_CALM_XGBoost_N30_t1", "algo": "XGBoost", "regime": "CALM", "horizon": 10, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "GILD_Gilead_ret_20d", "CI_Cigna_vol_20d", "IWM_SmallCap_vol_20d", "EWS_Singapore_ret_5d", "AXP_Amex_ret_20d", "INTC_ret_1d", "EWA_Australia_ret_1d", "TED_Spread_zscore_60d", "Nikkei_Japan_zscore_60d", "EWQ_France_zscore_60d", "Retail_Sales_zscore_60d", "PG_ret_20d", "ENB_EnbridgeInc_ret_1d", "MRK_Merck_zscore_60d", "EWL_Switzerland_vol_20d", "VVIX_ret_20d", "EMR_Emerson_ret_20d", "US3M_Rate_vol_20d", "BTI_BritishAmerican_ret_20d", "HangSeng_HK_ret_5d", "ORCL_vol_20d", "NWL_Newell_ret_20d", "TED_Spread_vol_20d", "LLY_zscore_60d", "Brent_Oil_FRED_ret_5d", "DHR_vol_20d", "Michigan_Sentiment_ret_20d", "SBUX_vol_20d"], "is_new": true}, {"model_id": "new_h10_CALM_XGBoost_N30_t2", "algo": "XGBoost", "regime": "CALM", "horizon": 10, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "vix_acceleration_1d", "EWM_Malaysia_vol_20d", "hmm_p_stress", "TED_Spread_zscore_60d", "EWA_Australia_zscore_60d", "SBUX_vol_20d", "EWY_Korea_zscore_60d", "AVB_AvalonBay_zscore_60d", "EFFR_vol_20d", "LUV_SouthwestAir_ret_5d", "IWM_SmallCap_vol_20d", "EOG_EOGResources_vol_20d", "TGT_Target_zscore_60d", "GE_ret_1d", "ASX_Australia_ret_5d", "BDX_Becton_Dickinson_ret_20d", "EWL_Switzerland_zscore_60d", "LOW_Lowes_ret_5d", "EWL_Switzerland_vol_20d", "DHR_vol_20d", "GD_GeneralDynamics_zscore_60d", "HD_ret_20d", "INTC_ret_5d", "ORCL_zscore_60d", "NEE_NextEra_ret_20d", "SJM_JM_Smucker_ret_5d", "XOM_ret_1d", "EWY_Korea_ret_20d"], "is_new": true}, {"model_id": "new_h10_CALM_XGBoost_N30_t3", "algo": "XGBoost", "regime": "CALM", "horizon": 10, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "MS_MorganStanley_zscore_60d", "ASX_Australia_vol_20d", "SJM_JM_Smucker_ret_1d", "Retail_Sales_zscore_60d", "EWA_Australia_ret_1d", "T10Y2Y_Spread_ret_5d", "Nikkei_Japan_vol_20d", "MS_MorganStanley_ret_5d", "PCAR_PaccarInc_ret_5d", "EWA_Australia_zscore_60d", "PG_ret_20d", "SO_SouthernCo_ret_5d", "TGT_Target_zscore_60d", "NEE_NextEra_ret_20d", "TXN_vol_20d", "MSTR_Bitcoin3_ret_5d", "PAYX_Paychex_zscore_60d", "CPB_CampbellSoup_zscore_60d", "heston_var_ev_h3", "CPB_CampbellSoup_vol_20d", "heston_ev_h3", "ES_Evergy_ret_1d", "AMD_ret_5d", "DIS_vol_20d", "EWM_Malaysia_zscore_60d", "VOD_Vodafone_zscore_60d", "NFCI_ret_5d", "SBUX_zscore_60d"], "is_new": true}, {"model_id": "new_h10_CALM_XGBoost_N30_t4", "algo": "XGBoost", "regime": "CALM", "horizon": 10, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "M_Macys_vol_20d", "SJM_JM_Smucker_ret_1d", "DIS_vol_20d", "EWM_Malaysia_ret_1d", "EFFR_vol_20d", "EWJ_Japan_vol_20d", "ENB_EnbridgeInc_ret_1d", "US3Y_Rate_ret_5d", "heston_ev_h3", "3M_vol_20d", "PG_ret_20d", "Industrial_Production_zscore_60d", "Nikkei_Japan_zscore_60d", "TED_Spread_zscore_60d", "Core_CPI_zscore_60d", "US1Y_Rate_ret_5d", "US30Y_Rate_ret_20d", "Brent_Oil_FRED_ret_20d", "US3M_Rate_zscore_60d", "LMT_LockheedMartin_vol_20d", "ITT_ITTInc_ret_5d", "PPL_PPL_ret_1d", "PAYX_Paychex_vol_20d", "HD_ret_20d", "FedFunds_zscore_60d", "PAYX_Paychex_ret_20d", "MSTR_Bitcoin3_ret_5d", "NFCI_ret_5d"], "is_new": true}, {"model_id": "new_h10_CALM_XGBoost_N30_t5", "algo": "XGBoost", "regime": "CALM", "horizon": 10, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "XLF_Fin_vol_20d", "EOG_EOGResources_vol_20d", "Brent_Oil_FRED_ret_5d", "AMZN_ret_5d", "XLK_Tech_zscore_60d", "heston_var_ev_h7", "TM_Telephone_ret_1d", "EWQ_France_zscore_60d", "BLK_BlackRock_zscore_60d", "VVIX_ret_20d", "DHR_vol_20d", "HD_zscore_60d", "LOW_Lowes_ret_20d", "AXP_Amex_ret_20d", "3M_vol_20d", "HangSeng_HK_vol_20d", "M_Macys_vol_20d", "TGT_Target_zscore_60d", "SBUX_vol_20d", "BA_ret_1d", "T10Y2Y_Spread_ret_5d", "US3Y_Rate_ret_5d", "EWM_Malaysia_vol_20d", "ORCL_vol_20d", "CPB_CampbellSoup_ret_5d", "MS_MorganStanley_ret_5d", "DAX_Germany_zscore_60d", "EMR_Emerson_ret_20d"], "is_new": true}, {"model_id": "new_h10_CALM_XGBoost_N30_t6", "algo": "XGBoost", "regime": "CALM", "horizon": 10, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EQIX_Equinix_ret_5d", "Michigan_Sentiment_ret_20d", "CMCSA_ret_1d", "Nikkei_Japan_zscore_60d", "ORCL_vol_20d", "US3Y_Rate_ret_5d", "LMT_LockheedMartin_ret_1d", "EWJ_Japan_vol_20d", "HangSeng_HK_ret_1d", "EWL_Switzerland_zscore_60d", "EQR_Equity_ret_1d", "XOM_ret_1d", "Industrial_Production_zscore_60d", "BLK_BlackRock_zscore_60d", "MS_MorganStanley_ret_1d", "EXC_Exelon_zscore_60d", "MS_MorganStanley_ret_5d", "AMZN_ret_5d", "AORD_AUS_zscore_60d", "spx_vol_5d", "MO_AltriaMG_ret_1d", "LMT_LockheedMartin_vol_20d", "Core_PCE_zscore_60d", "MS_MorganStanley_zscore_60d", "spx_abs_ret_max_5d", "3M_vol_20d", "QQQ_vol_20d", "vix_mean_abs_ret_5d"], "is_new": true}, {"model_id": "new_h10_CALM_XGBoost_N30_t7", "algo": "XGBoost", "regime": "CALM", "horizon": 10, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "AXP_Amex_vol_20d", "VOD_Vodafone_zscore_60d", "SO_SouthernCo_ret_5d", "EWM_Malaysia_ret_1d", "ORCL_vol_20d", "NVDA_vol_20d", "HangSeng_HK_ret_5d", "CMCSA_ret_1d", "NOC_Northrop_ret_20d", "MRK_Merck_zscore_60d", "AMGN_Amgen_ret_1d", "QQQ_vol_20d", "MS_MorganStanley_ret_1d", "LUV_SouthwestAir_ret_5d", "MS_MorganStanley_ret_5d", "SBUX_ret_5d", "EWQ_France_zscore_60d", "TM_Telephone_ret_1d", "LMT_LockheedMartin_ret_1d", "PFE_ret_1d", "MO_AltriaMG_ret_1d", "BTI_BritishAmerican_ret_20d", "PAYX_Paychex_vol_20d", "CLX_Clorox_vol_20d", "HD_ret_20d", "LOW_Lowes_ret_20d", "EOG_EOGResources_ret_5d", "ENB_EnbridgeInc_ret_1d"], "is_new": true}, {"model_id": "new_h10_CALM_LightGBM_N5_t0", "algo": "LightGBM", "regime": "CALM", "horizon": 10, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "heston_var_ev_h7", "BDX_Becton_Dickinson_ret_20d", "US3Y_Rate_ret_5d"], "is_new": true}, {"model_id": "new_h10_CALM_LightGBM_N5_t1", "algo": "LightGBM", "regime": "CALM", "horizon": 10, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "HD_ret_20d", "AVB_AvalonBay_zscore_60d", "BDX_Becton_Dickinson_ret_20d"], "is_new": true}, {"model_id": "new_h10_CALM_LightGBM_N5_t2", "algo": "LightGBM", "regime": "CALM", "horizon": 10, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "US1Y_Rate_ret_20d", "Core_CPI_zscore_60d", "PG_ret_20d"], "is_new": true}, {"model_id": "new_h10_CALM_LightGBM_N5_t3", "algo": "LightGBM", "regime": "CALM", "horizon": 10, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "US1Y_Rate_ret_20d", "EWL_Switzerland_vol_20d", "PPL_PPL_ret_1d"], "is_new": true}, {"model_id": "new_h10_CALM_LightGBM_N5_t4", "algo": "LightGBM", "regime": "CALM", "horizon": 10, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "INTC_ret_1d", "SO_SouthernCo_ret_5d", "NWL_Newell_ret_20d"], "is_new": true}, {"model_id": "new_h10_CALM_LightGBM_N5_t5", "algo": "LightGBM", "regime": "CALM", "horizon": 10, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "SBUX_vol_20d", "XLB_Materials_zscore_60d", "DHR_vol_20d"], "is_new": true}, {"model_id": "new_h10_CALM_LightGBM_N5_t6", "algo": "LightGBM", "regime": "CALM", "horizon": 10, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "Retail_Sales_zscore_60d", "MS_MorganStanley_ret_5d", "US7Y_Rate_ret_20d"], "is_new": true}, {"model_id": "new_h10_CALM_LightGBM_N5_t7", "algo": "LightGBM", "regime": "CALM", "horizon": 10, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "BDX_Becton_Dickinson_ret_20d", "MSTR_Bitcoin3_ret_1d", "GE_ret_1d"], "is_new": true}, {"model_id": "new_h10_CALM_LightGBM_N8_t0", "algo": "LightGBM", "regime": "CALM", "horizon": 10, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWY_Korea_ret_20d", "VRP_ma5", "Nikkei_Japan_vol_20d", "EWM_Malaysia_zscore_60d", "VOD_Vodafone_zscore_60d", "DAX_Germany_vol_20d"], "is_new": true}, {"model_id": "new_h10_CALM_LightGBM_N8_t1", "algo": "LightGBM", "regime": "CALM", "horizon": 10, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "INTC_ret_5d", "DE_Deere_ret_5d", "Retail_Sales_zscore_60d", "MS_MorganStanley_zscore_60d", "SJM_JM_Smucker_ret_5d", "EWC_Canada_zscore_60d"], "is_new": true}, {"model_id": "new_h10_CALM_LightGBM_N8_t2", "algo": "LightGBM", "regime": "CALM", "horizon": 10, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "SPY_zscore_60d", "heston_var_ev_h7", "MS_MorganStanley_ret_1d", "Michigan_Sentiment_ret_20d", "US3Y_Rate_ret_5d", "CPB_CampbellSoup_ret_20d"], "is_new": true}, {"model_id": "new_h10_CALM_LightGBM_N8_t3", "algo": "LightGBM", "regime": "CALM", "horizon": 10, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "HangSeng_HK_vol_20d", "DHR_vol_20d", "IWM_SmallCap_vol_20d", "spx_abs_ret_max_5d", "ITT_ITTInc_ret_5d", "LLY_zscore_60d"], "is_new": true}, {"model_id": "new_h10_CALM_LightGBM_N8_t4", "algo": "LightGBM", "regime": "CALM", "horizon": 10, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWG_Germany_ret_20d", "US1Y_Rate_ret_5d", "SJM_JM_Smucker_ret_1d", "M_Macys_vol_20d", "ITT_ITTInc_ret_5d", "heston_ev_h3"], "is_new": true}, {"model_id": "new_h10_CALM_LightGBM_N8_t5", "algo": "LightGBM", "regime": "CALM", "horizon": 10, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "AORD_AUS_zscore_60d", "US1Y_Rate_ret_20d", "EWM_Malaysia_vol_20d", "FedFunds_zscore_60d", "CPB_CampbellSoup_ret_20d", "LMT_LockheedMartin_vol_20d"], "is_new": true}, {"model_id": "new_h10_CALM_LightGBM_N8_t6", "algo": "LightGBM", "regime": "CALM", "horizon": 10, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "MS_MorganStanley_ret_1d", "M_Macys_vol_20d", "AXP_Amex_vol_20d", "IYM_BasicMaterials_ret_20d", "MO_AltriaMG_ret_1d", "spx_vol_5d"], "is_new": true}, {"model_id": "new_h10_CALM_LightGBM_N8_t7", "algo": "LightGBM", "regime": "CALM", "horizon": 10, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "US3Y_Rate_ret_5d", "CPB_CampbellSoup_vol_20d", "IWM_SmallCap_vol_20d", "US3M_Rate_zscore_60d", "LOW_Lowes_ret_20d", "AMGN_Amgen_ret_1d"], "is_new": true}, {"model_id": "new_h10_CALM_LightGBM_N10_t0", "algo": "LightGBM", "regime": "CALM", "horizon": 10, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "PCAR_PaccarInc_ret_5d", "SJM_JM_Smucker_ret_5d", "MRK_Merck_zscore_60d", "HD_zscore_60d", "SBUX_vol_20d", "GE_ret_1d", "DIS_vol_20d", "US3Y_Rate_ret_5d"], "is_new": true}, {"model_id": "new_h10_CALM_LightGBM_N10_t1", "algo": "LightGBM", "regime": "CALM", "horizon": 10, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "hmm_p_stress", "SPY_zscore_60d", "FedFunds_zscore_60d", "EFFR_ret_1d", "XLY_Disc_vol_20d", "IYM_BasicMaterials_ret_20d", "EQIX_Equinix_ret_5d", "HD_ret_20d"], "is_new": true}, {"model_id": "new_h10_CALM_LightGBM_N10_t2", "algo": "LightGBM", "regime": "CALM", "horizon": 10, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "MSTR_Bitcoin3_ret_1d", "US7Y_Rate_ret_20d", "MSTR_Bitcoin3_ret_5d", "CLX_Clorox_vol_20d", "WTI_Oil_FRED_zscore_60d", "gjr_condvar_h1", "CPB_CampbellSoup_ret_5d", "EQIX_Equinix_ret_5d"], "is_new": true}, {"model_id": "new_h10_CALM_LightGBM_N10_t3", "algo": "LightGBM", "regime": "CALM", "horizon": 10, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "SPY_zscore_60d", "EWY_Korea_ret_20d", "EWM_Malaysia_ret_1d", "INTC_ret_1d", "AMGN_Amgen_ret_1d", "IBEX_Spain_ret_20d", "ENB_EnbridgeInc_ret_1d", "LMT_LockheedMartin_ret_1d"], "is_new": true}, {"model_id": "new_h10_CALM_LightGBM_N10_t4", "algo": "LightGBM", "regime": "CALM", "horizon": 10, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "SLB_Schlumberger_ret_1d", "EWA_Australia_zscore_60d", "JNJ_ret_1d", "DAX_Germany_vol_20d", "BTI_BritishAmerican_ret_5d", "XLY_Disc_vol_20d", "BDX_Becton_Dickinson_ret_20d", "EWG_Germany_vol_20d"], "is_new": true}, {"model_id": "new_h10_CALM_LightGBM_N10_t5", "algo": "LightGBM", "regime": "CALM", "horizon": 10, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "LOW_Lowes_ret_5d", "M_Macys_vol_20d", "EWQ_France_ret_20d", "SO_SouthernCo_ret_5d", "US1Y_Rate_ret_20d", "LLY_zscore_60d", "MS_MorganStanley_ret_1d", "TM_Telephone_vol_20d"], "is_new": true}, {"model_id": "new_h10_CALM_LightGBM_N10_t6", "algo": "LightGBM", "regime": "CALM", "horizon": 10, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "LLY_zscore_60d", "LMT_LockheedMartin_vol_20d", "hmm_p_stress", "Nikkei_Japan_vol_20d", "spx_abs_ret_max_5d", "HangSeng_HK_ret_5d", "EOG_EOGResources_ret_5d", "XLY_Disc_vol_20d"], "is_new": true}, {"model_id": "new_h10_CALM_LightGBM_N10_t7", "algo": "LightGBM", "regime": "CALM", "horizon": 10, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWC_Canada_zscore_60d", "M_Macys_vol_20d", "US1Y_Rate_ret_20d", "BTI_BritishAmerican_ret_5d", "EWQ_France_zscore_60d", "CCI_CrownCastle_vol_20d", "GE_ret_1d", "EWS_Singapore_ret_5d"], "is_new": true}, {"model_id": "new_h10_CALM_LightGBM_N12_t0", "algo": "LightGBM", "regime": "CALM", "horizon": 10, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EOG_EOGResources_ret_5d", "IBEX_Spain_ret_20d", "XLV_Health_zscore_60d", "EWM_Malaysia_ret_1d", "XLF_Fin_vol_20d", "PCAR_PaccarInc_ret_5d", "Nikkei_Japan_zscore_60d", "TED_Spread_vol_20d", "EXC_Exelon_zscore_60d", "vix_mean_abs_ret_5d"], "is_new": true}, {"model_id": "new_h10_CALM_LightGBM_N12_t1", "algo": "LightGBM", "regime": "CALM", "horizon": 10, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "AMD_ret_5d", "MSTR_Bitcoin3_ret_1d", "AVB_AvalonBay_zscore_60d", "gjr_condvar_h1", "EWA_Australia_zscore_60d", "DIS_vol_20d", "ES_Evergy_ret_1d", "XLK_Tech_zscore_60d", "US6M_Rate_ret_20d", "US30Y_Rate_ret_20d"], "is_new": true}, {"model_id": "new_h10_CALM_LightGBM_N12_t2", "algo": "LightGBM", "regime": "CALM", "horizon": 10, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "ITT_ITTInc_ret_5d", "MSTR_Bitcoin3_ret_20d", "Nikkei_Japan_vol_20d", "EFFR_ret_1d", "ES_Evergy_ret_1d", "TED_Spread_zscore_60d", "EOG_EOGResources_vol_20d", "heston_ev_h3", "LOW_Lowes_ret_20d", "CPB_CampbellSoup_vol_20d"], "is_new": true}, {"model_id": "new_h10_CALM_LightGBM_N12_t3", "algo": "LightGBM", "regime": "CALM", "horizon": 10, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "IYM_BasicMaterials_ret_20d", "EWM_Malaysia_ret_1d", "heston_var_ev_h7", "3M_ret_5d", "CLX_Clorox_vol_20d", "XLV_Health_zscore_60d", "EWA_Australia_ret_1d", "TM_Telephone_ret_1d", "HangSeng_HK_ret_1d", "EXC_Exelon_ret_1d"], "is_new": true}, {"model_id": "new_h10_CALM_LightGBM_N12_t4", "algo": "LightGBM", "regime": "CALM", "horizon": 10, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "LOW_Lowes_ret_20d", "CLX_Clorox_vol_20d", "LOW_Lowes_ret_5d", "EOG_EOGResources_ret_5d", "INTC_ret_1d", "NFCI_ret_5d", "MSTR_Bitcoin3_ret_5d", "EQR_Equity_ret_1d", "T_ret_1d", "SPY_zscore_60d"], "is_new": true}, {"model_id": "new_h10_CALM_LightGBM_N12_t5", "algo": "LightGBM", "regime": "CALM", "horizon": 10, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "AXP_Amex_ret_20d", "EQIX_Equinix_ret_5d", "CLX_Clorox_vol_20d", "NEE_NextEra_ret_20d", "SBUX_ret_5d", "EWA_Australia_zscore_60d", "TGT_Target_zscore_60d", "3M_ret_5d", "HangSeng_HK_ret_5d", "PAYX_Paychex_zscore_60d"], "is_new": true}, {"model_id": "new_h10_CALM_LightGBM_N12_t6", "algo": "LightGBM", "regime": "CALM", "horizon": 10, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "XLF_Fin_vol_20d", "BDX_Becton_Dickinson_ret_20d", "CI_Cigna_vol_20d", "SJM_JM_Smucker_ret_5d", "EWH_HongKong_ret_5d", "TED_Spread_zscore_60d", "HD_ret_20d", "HD_ret_1d", "EWM_Malaysia_vol_20d", "XOM_ret_1d"], "is_new": true}, {"model_id": "new_h10_CALM_LightGBM_N12_t7", "algo": "LightGBM", "regime": "CALM", "horizon": 10, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "XOM_ret_20d", "HUM_Humana_ret_5d", "XLV_Health_zscore_60d", "BA_ret_1d", "ES_Evergy_ret_1d", "EQIX_Equinix_ret_5d", "IYM_BasicMaterials_ret_20d", "IYR_US_REIT2_zscore_60d", "WTI_Oil_FRED_zscore_60d", "BDX_Becton_Dickinson_ret_20d"], "is_new": true}, {"model_id": "new_h10_CALM_LightGBM_N15_t0", "algo": "LightGBM", "regime": "CALM", "horizon": 10, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "DHR_ret_1d", "DAX_Germany_zscore_60d", "EWJ_Japan_vol_20d", "heston_var_ev_h3", "gjr_condvar_h1", "LMT_LockheedMartin_vol_20d", "spx_abs_ret_max_5d", "EWM_Malaysia_zscore_60d", "AXP_Amex_vol_20d", "EWM_Malaysia_ret_1d", "EWL_Switzerland_zscore_60d", "NFCI_ret_5d", "SBUX_ret_5d"], "is_new": true}, {"model_id": "new_h10_CALM_LightGBM_N15_t1", "algo": "LightGBM", "regime": "CALM", "horizon": 10, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "MS_MorganStanley_ret_1d", "NOC_Northrop_ret_20d", "CCI_CrownCastle_vol_20d", "DE_Deere_ret_5d", "MSTR_Bitcoin3_ret_5d", "heston_var_ev_h7", "T10Y2Y_Spread_ret_5d", "CPB_CampbellSoup_zscore_60d", "CTAS_Cintas_vol_20d", "XOM_ret_20d", "TGT_Target_zscore_60d", "EQR_Equity_ret_1d", "spx_abs_ret_max_5d"], "is_new": true}, {"model_id": "new_h10_CALM_LightGBM_N15_t2", "algo": "LightGBM", "regime": "CALM", "horizon": 10, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EFFR_vol_20d", "IYM_BasicMaterials_ret_20d", "ITT_ITTInc_ret_5d", "PCAR_PaccarInc_ret_5d", "hmm_p_stress", "HD_zscore_60d", "HUM_Humana_ret_5d", "HangSeng_HK_vol_20d", "IYR_US_REIT2_zscore_60d", "gjr_condvar_h1", "NVDA_vol_20d", "US6M_Rate_ret_20d", "EWQ_France_ret_20d"], "is_new": true}, {"model_id": "new_h10_CALM_LightGBM_N15_t3", "algo": "LightGBM", "regime": "CALM", "horizon": 10, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "Brent_Oil_FRED_ret_5d", "BLK_BlackRock_zscore_60d", "EWC_Canada_zscore_60d", "ENB_EnbridgeInc_ret_1d", "ORCL_vol_20d", "PG_ret_20d", "US1Y_Rate_ret_20d", "T_ret_1d", "PCAR_PaccarInc_ret_5d", "WTI_Oil_FRED_zscore_60d", "NWL_Newell_ret_20d", "NVDA_vol_20d", "Core_PCE_zscore_60d"], "is_new": true}, {"model_id": "new_h10_CALM_LightGBM_N15_t4", "algo": "LightGBM", "regime": "CALM", "horizon": 10, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWY_Korea_zscore_60d", "HD_ret_20d", "HangSeng_HK_ret_1d", "HD_zscore_60d", "Core_CPI_zscore_60d", "AXP_Amex_vol_20d", "HangSeng_HK_vol_20d", "US30Y_Rate_ret_20d", "NOC_Northrop_ret_20d", "TED_Spread_zscore_60d", "XLY_Disc_vol_20d", "AMT_AmericanTower_ret_1d", "Nikkei_Japan_vol_20d"], "is_new": true}, {"model_id": "new_h10_CALM_LightGBM_N15_t5", "algo": "LightGBM", "regime": "CALM", "horizon": 10, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "BTI_BritishAmerican_ret_5d", "FedFunds_zscore_60d", "EXC_Exelon_ret_1d", "TED_Spread_zscore_60d", "VRP_ma5", "BA_ret_1d", "MS_MorganStanley_ret_1d", "INTC_ret_1d", "US6M_Rate_ret_20d", "EWM_Malaysia_vol_20d", "PCAR_PaccarInc_ret_5d", "XOM_ret_1d", "EWM_Malaysia_ret_1d"], "is_new": true}, {"model_id": "new_h10_CALM_LightGBM_N15_t6", "algo": "LightGBM", "regime": "CALM", "horizon": 10, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "BLK_BlackRock_zscore_60d", "IYR_US_REIT2_zscore_60d", "PFE_ret_1d", "CPB_CampbellSoup_zscore_60d", "Nikkei_Japan_zscore_60d", "US3M_Rate_zscore_60d", "XLF_Fin_vol_20d", "HangSeng_HK_ret_5d", "EWM_Malaysia_ret_1d", "US3M_Rate_vol_20d", "CLX_Clorox_vol_20d", "TED_Spread_vol_20d", "IWM_SmallCap_vol_20d"], "is_new": true}, {"model_id": "new_h10_CALM_LightGBM_N15_t7", "algo": "LightGBM", "regime": "CALM", "horizon": 10, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "hmm_p_stress", "TXN_vol_20d", "spx_abs_ret_max_5d", "HD_ret_20d", "PAYX_Paychex_vol_20d", "IYM_BasicMaterials_ret_20d", "SPY_zscore_60d", "MS_MorganStanley_ret_5d", "DIS_vol_20d", "CMCSA_ret_1d", "TED_Spread_vol_20d", "SBUX_zscore_60d", "AMT_AmericanTower_ret_1d"], "is_new": true}, {"model_id": "new_h10_CALM_LightGBM_N20_t0", "algo": "LightGBM", "regime": "CALM", "horizon": 10, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "3M_vol_20d", "DOW_Price_zscore_60d", "EFFR_ret_1d", "HUM_Humana_ret_5d", "IYR_US_REIT2_zscore_60d", "US1Y_Rate_ret_5d", "LLY_zscore_60d", "PCAR_PaccarInc_ret_5d", "SCHW_Schwab_ret_5d", "DAX_Germany_vol_20d", "EWQ_France_ret_20d", "AMD_ret_5d", "hmm_p_stress", "LMT_LockheedMartin_ret_1d", "XLK_Tech_zscore_60d", "PPL_PPL_ret_1d", "Nikkei_Japan_vol_20d", "CPB_CampbellSoup_zscore_60d"], "is_new": true}, {"model_id": "new_h10_CALM_LightGBM_N20_t1", "algo": "LightGBM", "regime": "CALM", "horizon": 10, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "spx_vol_5d", "MRK_Merck_zscore_60d", "DAX_Germany_vol_20d", "EWH_HongKong_ret_5d", "EWA_Australia_ret_1d", "HUM_Humana_ret_5d", "CPB_CampbellSoup_ret_5d", "EOG_EOGResources_ret_5d", "EWC_Canada_zscore_60d", "BTI_BritishAmerican_ret_20d", "LOW_Lowes_ret_20d", "AXP_Amex_vol_20d", "EWY_Korea_ret_20d", "QQQ_vol_20d", "heston_var_ev_h5", "SBUX_vol_20d", "SBUX_ret_5d", "vix_acceleration_1d"], "is_new": true}, {"model_id": "new_h10_CALM_LightGBM_N20_t2", "algo": "LightGBM", "regime": "CALM", "horizon": 10, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "ORCL_vol_20d", "QQQ_vol_20d", "EWQ_France_ret_20d", "SPY_zscore_60d", "XLF_Fin_vol_20d", "NOC_Northrop_ret_20d", "AMT_AmericanTower_ret_1d", "AMD_ret_1d", "AXP_Amex_vol_20d", "EWA_Australia_zscore_60d", "DAX_Germany_zscore_60d", "ASX_Australia_ret_5d", "MSTR_Bitcoin3_ret_1d", "MS_MorganStanley_zscore_60d", "EWM_Malaysia_vol_20d", "Brent_Oil_FRED_ret_5d", "gjr_condvar_h1", "PLD_Prologis_ret_5d"], "is_new": true}, {"model_id": "new_h10_CALM_LightGBM_N20_t3", "algo": "LightGBM", "regime": "CALM", "horizon": 10, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "HangSeng_HK_ret_1d", "HD_ret_20d", "DAX_Germany_zscore_60d", "HD_ret_1d", "Core_PCE_zscore_60d", "MSTR_Bitcoin3_ret_20d", "NVDA_vol_20d", "AXP_Amex_ret_20d", "SBUX_vol_20d", "EOG_EOGResources_ret_5d", "VVIX_ret_20d", "XLY_Disc_vol_20d", "SO_SouthernCo_ret_5d", "VOD_Vodafone_zscore_60d", "LMT_LockheedMartin_vol_20d", "Industrial_Production_zscore_60d", "EWA_Australia_zscore_60d", "HD_ret_5d"], "is_new": true}, {"model_id": "new_h10_CALM_LightGBM_N20_t4", "algo": "LightGBM", "regime": "CALM", "horizon": 10, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "TED_Spread_zscore_60d", "EWM_Malaysia_ret_1d", "HUM_Humana_ret_5d", "EQR_Equity_ret_1d", "heston_var_ev_h7", "DHR_vol_20d", "PAYX_Paychex_vol_20d", "Retail_Sales_zscore_60d", "TED_Spread_vol_20d", "WTI_Oil_FRED_zscore_60d", "EWG_Germany_ret_20d", "EXC_Exelon_zscore_60d", "EWY_Korea_zscore_60d", "heston_var_ev_h3", "EWM_Malaysia_vol_20d", "EWL_Switzerland_vol_20d", "EWA_Australia_zscore_60d", "Michigan_Sentiment_ret_20d"], "is_new": true}, {"model_id": "new_h10_CALM_LightGBM_N20_t5", "algo": "LightGBM", "regime": "CALM", "horizon": 10, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "AMT_AmericanTower_ret_1d", "CPB_CampbellSoup_ret_5d", "VVIX_ret_20d", "AORD_AUS_zscore_60d", "US5Y_Rate_ret_5d", "EWG_Germany_vol_20d", "US1Y_Rate_ret_5d", "QQQ_vol_20d", "hmm_p_stress", "NEE_NextEra_ret_20d", "spx_momentum_3d", "CI_Cigna_vol_20d", "XOM_ret_20d", "spx_vol_5d", "IBEX_Spain_ret_20d", "PG_ret_20d", "EWA_Australia_ret_1d", "EWH_HongKong_ret_5d"], "is_new": true}, {"model_id": "new_h10_CALM_LightGBM_N20_t6", "algo": "LightGBM", "regime": "CALM", "horizon": 10, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "vix_mean_abs_ret_5d", "EWG_Germany_ret_20d", "heston_var_ev_h3", "EWM_Malaysia_ret_1d", "US3Y_Rate_ret_5d", "XLK_Tech_zscore_60d", "MRK_Merck_zscore_60d", "spx_momentum_3d", "HD_ret_1d", "CPB_CampbellSoup_vol_20d", "3M_vol_20d", "EWQ_France_ret_20d", "EXC_Exelon_zscore_60d", "LOW_Lowes_ret_20d", "LMT_LockheedMartin_ret_1d", "LLY_zscore_60d", "BDX_Becton_Dickinson_ret_20d", "CPB_CampbellSoup_zscore_60d"], "is_new": true}, {"model_id": "new_h10_CALM_LightGBM_N20_t7", "algo": "LightGBM", "regime": "CALM", "horizon": 10, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "SBUX_vol_20d", "spx_vol_5d", "hmm_p_stress", "gjr_condvar_h1", "EWL_Switzerland_vol_20d", "BA_ret_1d", "AORD_AUS_zscore_60d", "EOG_EOGResources_vol_20d", "DE_Deere_vol_20d", "VRP_ma5", "3M_ret_5d", "vix_mean_abs_ret_5d", "ORCL_zscore_60d", "EWS_Singapore_ret_5d", "Retail_Sales_zscore_60d", "spx_momentum_3d", "Nikkei_Japan_zscore_60d", "IWM_SmallCap_vol_20d"], "is_new": true}, {"model_id": "new_h10_CALM_LightGBM_N25_t0", "algo": "LightGBM", "regime": "CALM", "horizon": 10, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "HangSeng_HK_vol_20d", "MS_MorganStanley_ret_5d", "ES_Evergy_ret_1d", "EOG_EOGResources_vol_20d", "AMD_ret_5d", "EWA_Australia_zscore_60d", "M_Macys_vol_20d", "XLB_Materials_zscore_60d", "WTI_Oil_FRED_zscore_60d", "VRP_ma5", "IYR_US_REIT2_zscore_60d", "QQQ_vol_20d", "LLY_zscore_60d", "EWS_Singapore_ret_5d", "SPY_zscore_60d", "AXP_Amex_ret_20d", "EQIX_Equinix_ret_5d", "US7Y_Rate_ret_20d", "heston_ev_h3", "CPB_CampbellSoup_vol_20d", "spx_vol_5d", "AMT_AmericanTower_ret_1d", "DE_Deere_ret_5d"], "is_new": true}, {"model_id": "new_h10_CALM_LightGBM_N25_t1", "algo": "LightGBM", "regime": "CALM", "horizon": 10, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWM_Malaysia_vol_20d", "EOG_EOGResources_vol_20d", "Michigan_Sentiment_ret_20d", "GILD_Gilead_ret_20d", "3M_vol_20d", "CCI_CrownCastle_vol_20d", "CPB_CampbellSoup_zscore_60d", "MS_MorganStanley_ret_5d", "NOC_Northrop_ret_20d", "DHR_ret_1d", "SO_SouthernCo_ret_5d", "XLY_Disc_vol_20d", "DOW_Price_zscore_60d", "PAYX_Paychex_ret_20d", "US3Y_Rate_ret_5d", "EWQ_France_zscore_60d", "BLK_BlackRock_zscore_60d", "TM_Telephone_vol_20d", "CMCSA_ret_1d", "LOW_Lowes_ret_20d", "IWM_SmallCap_vol_20d", "LMT_LockheedMartin_vol_20d", "ASX_Australia_ret_5d"], "is_new": true}, {"model_id": "new_h10_CALM_LightGBM_N25_t2", "algo": "LightGBM", "regime": "CALM", "horizon": 10, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "AXP_Amex_vol_20d", "TED_Spread_zscore_60d", "IWM_SmallCap_vol_20d", "SO_SouthernCo_ret_5d", "LOW_Lowes_ret_5d", "AMT_AmericanTower_ret_1d", "BA_ret_1d", "Core_CPI_zscore_60d", "PAYX_Paychex_vol_20d", "Nikkei_Japan_vol_20d", "US5Y_Rate_ret_5d", "EOG_EOGResources_ret_5d", "ASX_Australia_ret_5d", "US7Y_Rate_ret_20d", "AXP_Amex_ret_20d", "PG_ret_20d", "EFFR_vol_20d", "PFE_ret_1d", "TXN_vol_20d", "3M_ret_5d", "AMD_ret_5d", "SLB_Schlumberger_ret_5d", "spx_abs_ret_max_5d"], "is_new": true}, {"model_id": "new_h10_CALM_LightGBM_N25_t3", "algo": "LightGBM", "regime": "CALM", "horizon": 10, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWC_Canada_zscore_60d", "ASX_Australia_vol_20d", "TED_Spread_zscore_60d", "US3Y_Rate_ret_5d", "Nikkei_Japan_vol_20d", "HangSeng_HK_vol_20d", "ORCL_vol_20d", "spx_momentum_3d", "MSTR_Bitcoin3_ret_1d", "ES_Evergy_ret_1d", "HD_ret_5d", "EWA_Australia_ret_1d", "GILD_Gilead_ret_20d", "CTAS_Cintas_vol_20d", "EWQ_France_ret_20d", "FedFunds_zscore_60d", "US3M_Rate_vol_20d", "DIS_vol_20d", "IYR_US_REIT2_zscore_60d", "LOW_Lowes_ret_20d", "HD_zscore_60d", "VOD_Vodafone_zscore_60d", "LMT_LockheedMartin_vol_20d"], "is_new": true}, {"model_id": "new_h10_CALM_LightGBM_N25_t4", "algo": "LightGBM", "regime": "CALM", "horizon": 10, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "CMCSA_ret_1d", "Nikkei_Japan_zscore_60d", "SBUX_zscore_60d", "DE_Deere_vol_20d", "DE_Deere_ret_5d", "EWS_Singapore_ret_5d", "HangSeng_HK_ret_5d", "AORD_AUS_zscore_60d", "EQIX_Equinix_ret_5d", "T_ret_1d", "PFE_ret_1d", "VOD_Vodafone_zscore_60d", "LOW_Lowes_ret_5d", "gjr_condvar_h1", "EWA_Australia_ret_1d", "NWL_Newell_ret_20d", "EWQ_France_zscore_60d", "PCAR_PaccarInc_ret_5d", "INTC_ret_1d", "US3M_Rate_zscore_60d", "IWM_SmallCap_vol_20d", "EWL_Switzerland_zscore_60d", "spx_momentum_3d"], "is_new": true}, {"model_id": "new_h10_CALM_LightGBM_N25_t5", "algo": "LightGBM", "regime": "CALM", "horizon": 10, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "SCHW_Schwab_ret_5d", "MSTR_Bitcoin3_ret_20d", "Brent_Oil_FRED_ret_20d", "EWL_Switzerland_vol_20d", "NEE_NextEra_ret_20d", "CI_Cigna_vol_20d", "PFE_ret_1d", "EWM_Malaysia_vol_20d", "spx_abs_ret_max_5d", "US3M_Rate_vol_20d", "heston_var_ev_h3", "Industrial_Production_zscore_60d", "EXC_Exelon_ret_1d", "BDX_Becton_Dickinson_ret_20d", "EWG_Germany_ret_20d", "ITT_ITTInc_ret_5d", "Nikkei_Japan_zscore_60d", "SO_SouthernCo_ret_5d", "heston_var_ev_h7", "HangSeng_HK_ret_1d", "DE_Deere_vol_20d", "Brent_Oil_FRED_ret_5d", "DHR_ret_1d"], "is_new": true}, {"model_id": "new_h10_CALM_LightGBM_N25_t6", "algo": "LightGBM", "regime": "CALM", "horizon": 10, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWC_Canada_zscore_60d", "CLX_Clorox_vol_20d", "CCI_CrownCastle_vol_20d", "EWM_Malaysia_zscore_60d", "ASX_Australia_ret_5d", "CMCSA_ret_1d", "DIS_vol_20d", "Brent_Oil_FRED_ret_20d", "LOW_Lowes_ret_20d", "LUV_SouthwestAir_ret_5d", "AXP_Amex_ret_20d", "ASX_Australia_vol_20d", "Core_CPI_zscore_60d", "QQQ_vol_20d", "US3Y_Rate_ret_5d", "EMR_Emerson_ret_20d", "DHR_ret_1d", "IWM_SmallCap_vol_20d", "Nikkei_Japan_vol_20d", "NFCI_ret_5d", "GE_ret_1d", "Brent_Oil_FRED_ret_5d", "TED_Spread_vol_20d"], "is_new": true}, {"model_id": "new_h10_CALM_LightGBM_N25_t7", "algo": "LightGBM", "regime": "CALM", "horizon": 10, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "T_ret_1d", "vix_mean_abs_ret_5d", "ASX_Australia_ret_5d", "MSTR_Bitcoin3_ret_20d", "HD_ret_1d", "heston_var_ev_h3", "AXP_Amex_ret_20d", "MS_MorganStanley_ret_5d", "SPY_zscore_60d", "IWM_SmallCap_vol_20d", "TXN_vol_20d", "AMD_ret_1d", "MSTR_Bitcoin3_ret_5d", "VVIX_ret_20d", "US30Y_Rate_ret_20d", "GD_GeneralDynamics_zscore_60d", "EWQ_France_ret_20d", "HangSeng_HK_ret_1d", "VOD_Vodafone_zscore_60d", "LOW_Lowes_ret_5d", "SBUX_ret_5d", "LMT_LockheedMartin_vol_20d", "hmm_p_stress"], "is_new": true}, {"model_id": "new_h10_CALM_LightGBM_N30_t0", "algo": "LightGBM", "regime": "CALM", "horizon": 10, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "PPL_PPL_ret_1d", "BTI_BritishAmerican_ret_5d", "CMCSA_ret_1d", "GE_ret_1d", "ORCL_vol_20d", "DAX_Germany_zscore_60d", "gjr_condvar_h1", "PAYX_Paychex_vol_20d", "QQQ_vol_20d", "DHR_vol_20d", "3M_ret_5d", "TXN_vol_20d", "AXP_Amex_vol_20d", "PFE_ret_1d", "VRP_ma5", "HangSeng_HK_ret_1d", "XOM_ret_20d", "EWC_Canada_zscore_60d", "INTC_ret_5d", "MO_AltriaMG_ret_1d", "EWA_Australia_zscore_60d", "XLF_Fin_vol_20d", "CCI_CrownCastle_vol_20d", "AMD_ret_5d", "MS_MorganStanley_zscore_60d", "Nikkei_Japan_zscore_60d", "US5Y_Rate_ret_5d", "CLX_Clorox_vol_20d"], "is_new": true}, {"model_id": "new_h10_CALM_LightGBM_N30_t1", "algo": "LightGBM", "regime": "CALM", "horizon": 10, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "DAX_Germany_vol_20d", "EWL_Switzerland_zscore_60d", "ES_Evergy_ret_1d", "AMZN_ret_5d", "US3M_Rate_zscore_60d", "MS_MorganStanley_zscore_60d", "EWG_Germany_vol_20d", "heston_ev_h3", "INTC_ret_5d", "TXN_vol_20d", "PCAR_PaccarInc_ret_5d", "US5Y_Rate_ret_5d", "EFFR_ret_1d", "Brent_Oil_FRED_ret_20d", "DHR_vol_20d", "IYM_BasicMaterials_ret_20d", "CLX_Clorox_vol_20d", "NWL_Newell_ret_20d", "AMD_ret_5d", "EWM_Malaysia_zscore_60d", "spx_abs_ret_max_5d", "T_ret_1d", "heston_var_ev_h5", "spx_momentum_3d", "LOW_Lowes_ret_5d", "ENB_EnbridgeInc_ret_1d", "CPB_CampbellSoup_ret_5d", "Industrial_Production_zscore_60d"], "is_new": true}, {"model_id": "new_h10_CALM_LightGBM_N30_t2", "algo": "LightGBM", "regime": "CALM", "horizon": 10, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "CTAS_Cintas_vol_20d", "XLK_Tech_zscore_60d", "EQR_Equity_ret_1d", "DOW_Price_zscore_60d", "US6M_Rate_ret_20d", "CCI_CrownCastle_vol_20d", "INTC_ret_1d", "EWC_Canada_zscore_60d", "GD_GeneralDynamics_zscore_60d", "XOM_ret_20d", "ORCL_vol_20d", "CI_Cigna_vol_20d", "XLV_Health_zscore_60d", "CLX_Clorox_vol_20d", "VVIX_ret_20d", "TM_Telephone_ret_1d", "PAYX_Paychex_vol_20d", "TED_Spread_vol_20d", "PFE_ret_1d", "IYM_BasicMaterials_ret_20d", "US1Y_Rate_ret_5d", "DE_Deere_ret_5d", "XLF_Fin_vol_20d", "EWA_Australia_zscore_60d", "IYR_US_REIT2_zscore_60d", "NFCI_ret_5d", "EQIX_Equinix_ret_5d", "EWG_Germany_vol_20d"], "is_new": true}, {"model_id": "new_h10_CALM_LightGBM_N30_t3", "algo": "LightGBM", "regime": "CALM", "horizon": 10, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "AMD_ret_1d", "EQR_Equity_ret_1d", "PPL_PPL_ret_1d", "ASX_Australia_ret_5d", "EWG_Germany_ret_20d", "DAX_Germany_vol_20d", "HangSeng_HK_vol_20d", "HangSeng_HK_ret_1d", "EFFR_vol_20d", "LLY_zscore_60d", "CTAS_Cintas_vol_20d", "BDX_Becton_Dickinson_ret_20d", "gjr_condvar_h1", "CLX_Clorox_vol_20d", "AXP_Amex_ret_20d", "WTI_Oil_FRED_zscore_60d", "vix_mean_abs_ret_5d", "3M_ret_5d", "LUV_SouthwestAir_ret_5d", "EWG_Germany_vol_20d", "ORCL_zscore_60d", "AXP_Amex_vol_20d", "US3M_Rate_vol_20d", "SLB_Schlumberger_ret_5d", "MSTR_Bitcoin3_ret_20d", "Core_PCE_zscore_60d", "Retail_Sales_zscore_60d", "DAX_Germany_zscore_60d"], "is_new": true}, {"model_id": "new_h10_CALM_LightGBM_N30_t4", "algo": "LightGBM", "regime": "CALM", "horizon": 10, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "DE_Deere_ret_5d", "spx_abs_ret_max_5d", "US6M_Rate_ret_20d", "TED_Spread_zscore_60d", "EWY_Korea_ret_20d", "SBUX_zscore_60d", "HUM_Humana_ret_5d", "NVDA_vol_20d", "Brent_Oil_FRED_ret_5d", "EOG_EOGResources_vol_20d", "EOG_EOGResources_ret_5d", "AMZN_ret_5d", "PG_ret_20d", "BA_ret_1d", "PAYX_Paychex_zscore_60d", "EWG_Germany_vol_20d", "Brent_Oil_FRED_ret_20d", "hmm_p_stress", "BTI_BritishAmerican_ret_20d", "Retail_Sales_zscore_60d", "ITT_ITTInc_ret_5d", "PLD_Prologis_ret_5d", "MS_MorganStanley_zscore_60d", "EQR_Equity_ret_1d", "SO_SouthernCo_ret_5d", "EWL_Switzerland_vol_20d", "ORCL_zscore_60d", "MSTR_Bitcoin3_ret_1d"], "is_new": true}, {"model_id": "new_h10_CALM_LightGBM_N30_t5", "algo": "LightGBM", "regime": "CALM", "horizon": 10, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWL_Switzerland_zscore_60d", "PCAR_PaccarInc_ret_5d", "MS_MorganStanley_zscore_60d", "EWQ_France_ret_20d", "PAYX_Paychex_zscore_60d", "EWG_Germany_ret_20d", "EWA_Australia_zscore_60d", "IYR_US_REIT2_zscore_60d", "EWY_Korea_zscore_60d", "PLD_Prologis_ret_5d", "DHR_vol_20d", "HD_ret_1d", "AORD_AUS_zscore_60d", "BTI_BritishAmerican_ret_20d", "TED_Spread_vol_20d", "WTI_Oil_FRED_zscore_60d", "BTI_BritishAmerican_ret_5d", "LOW_Lowes_ret_5d", "CPB_CampbellSoup_ret_5d", "AMD_ret_1d", "US1Y_Rate_ret_5d", "Industrial_Production_zscore_60d", "NOC_Northrop_ret_20d", "TGT_Target_zscore_60d", "SPY_zscore_60d", "MSTR_Bitcoin3_ret_20d", "CTAS_Cintas_vol_20d", "IYM_BasicMaterials_ret_20d"], "is_new": true}, {"model_id": "new_h10_CALM_LightGBM_N30_t6", "algo": "LightGBM", "regime": "CALM", "horizon": 10, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "Brent_Oil_FRED_ret_5d", "EWQ_France_ret_20d", "ITT_ITTInc_ret_5d", "US3Y_Rate_ret_5d", "EMR_Emerson_ret_20d", "BLK_BlackRock_zscore_60d", "PFE_ret_1d", "MS_MorganStanley_zscore_60d", "NOC_Northrop_ret_20d", "SBUX_vol_20d", "VVIX_ret_20d", "T10Y2Y_Spread_ret_5d", "XOM_ret_20d", "VRP_ma5", "NEE_NextEra_ret_20d", "SLB_Schlumberger_ret_5d", "EXC_Exelon_ret_1d", "DE_Deere_vol_20d", "SCHW_Schwab_ret_5d", "IYM_BasicMaterials_ret_20d", "US1Y_Rate_ret_5d", "HD_zscore_60d", "MS_MorganStanley_ret_1d", "MS_MorganStanley_ret_5d", "DAX_Germany_zscore_60d", "EWC_Canada_zscore_60d", "HangSeng_HK_ret_5d", "GD_GeneralDynamics_zscore_60d"], "is_new": true}, {"model_id": "new_h10_CALM_LightGBM_N30_t7", "algo": "LightGBM", "regime": "CALM", "horizon": 10, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "PFE_ret_1d", "TXN_vol_20d", "PPL_PPL_ret_1d", "MS_MorganStanley_ret_1d", "CI_Cigna_vol_20d", "T10Y2Y_Spread_ret_5d", "SBUX_ret_5d", "3M_vol_20d", "LMT_LockheedMartin_ret_1d", "Retail_Sales_zscore_60d", "AORD_AUS_zscore_60d", "TED_Spread_vol_20d", "Core_PCE_zscore_60d", "BLK_BlackRock_zscore_60d", "US6M_Rate_ret_20d", "Brent_Oil_FRED_ret_5d", "EWH_HongKong_ret_5d", "HangSeng_HK_ret_5d", "EWC_Canada_zscore_60d", "XLB_Materials_zscore_60d", "DHR_vol_20d", "US7Y_Rate_ret_20d", "NVDA_vol_20d", "VOD_Vodafone_zscore_60d", "IBEX_Spain_ret_20d", "EWJ_Japan_vol_20d", "CPB_CampbellSoup_ret_20d", "EFFR_ret_1d"], "is_new": true}, {"model_id": "new_h10_CALM_GradientBoosting_N5_t0", "algo": "GradientBoosting", "regime": "CALM", "horizon": 10, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "CLX_Clorox_vol_20d", "spx_momentum_3d", "Michigan_Sentiment_ret_20d"], "is_new": true}, {"model_id": "new_h10_CALM_GradientBoosting_N5_t1", "algo": "GradientBoosting", "regime": "CALM", "horizon": 10, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "BA_ret_1d", "XLB_Materials_zscore_60d", "3M_ret_5d"], "is_new": true}, {"model_id": "new_h10_CALM_GradientBoosting_N5_t2", "algo": "GradientBoosting", "regime": "CALM", "horizon": 10, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EFFR_vol_20d", "TM_Telephone_ret_1d", "3M_vol_20d"], "is_new": true}, {"model_id": "new_h10_CALM_GradientBoosting_N5_t3", "algo": "GradientBoosting", "regime": "CALM", "horizon": 10, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "AMT_AmericanTower_ret_1d", "CPB_CampbellSoup_ret_5d", "3M_ret_5d"], "is_new": true}, {"model_id": "new_h10_CALM_GradientBoosting_N5_t4", "algo": "GradientBoosting", "regime": "CALM", "horizon": 10, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "FedFunds_zscore_60d", "TED_Spread_vol_20d", "EXC_Exelon_zscore_60d"], "is_new": true}, {"model_id": "new_h10_CALM_GradientBoosting_N5_t5", "algo": "GradientBoosting", "regime": "CALM", "horizon": 10, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "WTI_Oil_FRED_zscore_60d", "SBUX_ret_5d", "DAX_Germany_vol_20d"], "is_new": true}, {"model_id": "new_h10_CALM_GradientBoosting_N5_t6", "algo": "GradientBoosting", "regime": "CALM", "horizon": 10, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "XOM_ret_1d", "SCHW_Schwab_ret_5d", "HD_ret_20d"], "is_new": true}, {"model_id": "new_h10_CALM_GradientBoosting_N5_t7", "algo": "GradientBoosting", "regime": "CALM", "horizon": 10, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "ASX_Australia_vol_20d", "ENB_EnbridgeInc_ret_1d", "US3M_Rate_zscore_60d"], "is_new": true}, {"model_id": "new_h10_CALM_GradientBoosting_N8_t0", "algo": "GradientBoosting", "regime": "CALM", "horizon": 10, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "ES_Evergy_ret_1d", "EWL_Switzerland_vol_20d", "US6M_Rate_ret_20d", "EMR_Emerson_ret_20d", "US3M_Rate_zscore_60d", "BTI_BritishAmerican_ret_5d"], "is_new": true}, {"model_id": "new_h10_CALM_GradientBoosting_N8_t1", "algo": "GradientBoosting", "regime": "CALM", "horizon": 10, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWJ_Japan_vol_20d", "CLX_Clorox_vol_20d", "AVB_AvalonBay_zscore_60d", "Michigan_Sentiment_ret_20d", "AMT_AmericanTower_ret_1d", "AMD_ret_5d"], "is_new": true}, {"model_id": "new_h10_CALM_GradientBoosting_N8_t2", "algo": "GradientBoosting", "regime": "CALM", "horizon": 10, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "LOW_Lowes_ret_5d", "US7Y_Rate_ret_20d", "spx_abs_ret_max_5d", "heston_var_ev_h3", "GE_ret_1d", "TM_Telephone_vol_20d"], "is_new": true}, {"model_id": "new_h10_CALM_GradientBoosting_N8_t3", "algo": "GradientBoosting", "regime": "CALM", "horizon": 10, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "spx_abs_ret_max_5d", "XLV_Health_zscore_60d", "ITT_ITTInc_ret_5d", "MSTR_Bitcoin3_ret_1d", "EWQ_France_zscore_60d", "US3Y_Rate_ret_5d"], "is_new": true}, {"model_id": "new_h10_CALM_GradientBoosting_N8_t4", "algo": "GradientBoosting", "regime": "CALM", "horizon": 10, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "MS_MorganStanley_ret_5d", "ASX_Australia_ret_5d", "EXC_Exelon_ret_1d", "TGT_Target_zscore_60d", "gjr_condvar_h1", "Michigan_Sentiment_ret_20d"], "is_new": true}, {"model_id": "new_h10_CALM_GradientBoosting_N8_t5", "algo": "GradientBoosting", "regime": "CALM", "horizon": 10, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "Nikkei_Japan_vol_20d", "IYM_BasicMaterials_ret_20d", "TM_Telephone_ret_1d", "EQIX_Equinix_ret_5d", "TM_Telephone_vol_20d", "EQR_Equity_ret_1d"], "is_new": true}, {"model_id": "new_h10_CALM_GradientBoosting_N8_t6", "algo": "GradientBoosting", "regime": "CALM", "horizon": 10, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "XOM_ret_1d", "DAX_Germany_vol_20d", "gjr_condvar_h1", "MS_MorganStanley_zscore_60d", "CPB_CampbellSoup_vol_20d", "EWA_Australia_zscore_60d"], "is_new": true}, {"model_id": "new_h10_CALM_GradientBoosting_N8_t7", "algo": "GradientBoosting", "regime": "CALM", "horizon": 10, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "ITT_ITTInc_ret_5d", "US7Y_Rate_ret_20d", "HD_zscore_60d", "T_ret_1d", "EOG_EOGResources_ret_5d", "WTI_Oil_FRED_zscore_60d"], "is_new": true}, {"model_id": "new_h10_CALM_GradientBoosting_N10_t0", "algo": "GradientBoosting", "regime": "CALM", "horizon": 10, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "JNJ_ret_1d", "AMD_ret_1d", "CPB_CampbellSoup_vol_20d", "NEE_NextEra_ret_20d", "VOD_Vodafone_zscore_60d", "EWM_Malaysia_ret_1d", "XLB_Materials_zscore_60d", "EWQ_France_zscore_60d"], "is_new": true}, {"model_id": "new_h10_CALM_GradientBoosting_N10_t1", "algo": "GradientBoosting", "regime": "CALM", "horizon": 10, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "US3M_Rate_vol_20d", "AXP_Amex_ret_20d", "PFE_ret_1d", "EQR_Equity_ret_1d", "ORCL_vol_20d", "NWL_Newell_ret_20d", "heston_var_ev_h7", "vix_acceleration_1d"], "is_new": true}, {"model_id": "new_h10_CALM_GradientBoosting_N10_t2", "algo": "GradientBoosting", "regime": "CALM", "horizon": 10, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "DE_Deere_vol_20d", "vix_mean_abs_ret_5d", "INTC_ret_1d", "MSTR_Bitcoin3_ret_5d", "EFFR_ret_1d", "MSTR_Bitcoin3_ret_1d", "Nikkei_Japan_zscore_60d", "LOW_Lowes_ret_20d"], "is_new": true}, {"model_id": "new_h10_CALM_GradientBoosting_N10_t3", "algo": "GradientBoosting", "regime": "CALM", "horizon": 10, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "PFE_ret_1d", "HD_ret_20d", "MRK_Merck_zscore_60d", "SO_SouthernCo_ret_5d", "AMT_AmericanTower_ret_1d", "DIS_vol_20d", "SJM_JM_Smucker_ret_5d", "heston_var_ev_h5"], "is_new": true}, {"model_id": "new_h10_CALM_GradientBoosting_N10_t4", "algo": "GradientBoosting", "regime": "CALM", "horizon": 10, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "CPB_CampbellSoup_ret_5d", "ASX_Australia_ret_5d", "CI_Cigna_vol_20d", "VRP_ma5", "EOG_EOGResources_vol_20d", "EWA_Australia_ret_1d", "PLD_Prologis_ret_5d", "AXP_Amex_vol_20d"], "is_new": true}, {"model_id": "new_h10_CALM_GradientBoosting_N10_t5", "algo": "GradientBoosting", "regime": "CALM", "horizon": 10, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "WTI_Oil_FRED_zscore_60d", "Core_CPI_zscore_60d", "LUV_SouthwestAir_ret_5d", "SBUX_vol_20d", "EOG_EOGResources_ret_5d", "MS_MorganStanley_ret_1d", "JNJ_ret_1d", "EFFR_vol_20d"], "is_new": true}, {"model_id": "new_h10_CALM_GradientBoosting_N10_t6", "algo": "GradientBoosting", "regime": "CALM", "horizon": 10, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "LLY_zscore_60d", "EFFR_vol_20d", "FedFunds_zscore_60d", "EWH_HongKong_ret_5d", "US3M_Rate_zscore_60d", "HangSeng_HK_ret_1d", "EWG_Germany_vol_20d", "IYR_US_REIT2_zscore_60d"], "is_new": true}, {"model_id": "new_h10_CALM_GradientBoosting_N10_t7", "algo": "GradientBoosting", "regime": "CALM", "horizon": 10, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "Michigan_Sentiment_ret_20d", "LLY_zscore_60d", "TM_Telephone_ret_1d", "MSTR_Bitcoin3_ret_20d", "Brent_Oil_FRED_ret_5d", "Industrial_Production_zscore_60d", "SLB_Schlumberger_ret_1d", "EWY_Korea_zscore_60d"], "is_new": true}, {"model_id": "new_h10_CALM_GradientBoosting_N12_t0", "algo": "GradientBoosting", "regime": "CALM", "horizon": 10, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "XLY_Disc_vol_20d", "BTI_BritishAmerican_ret_20d", "SLB_Schlumberger_ret_1d", "TED_Spread_zscore_60d", "heston_ev_h3", "DHR_vol_20d", "HUM_Humana_ret_5d", "EWG_Germany_vol_20d", "ORCL_vol_20d", "SBUX_vol_20d"], "is_new": true}, {"model_id": "new_h10_CALM_GradientBoosting_N12_t1", "algo": "GradientBoosting", "regime": "CALM", "horizon": 10, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "NFCI_ret_5d", "EFFR_vol_20d", "ENB_EnbridgeInc_ret_1d", "SBUX_vol_20d", "SCHW_Schwab_ret_5d", "CI_Cigna_vol_20d", "EFFR_ret_1d", "CPB_CampbellSoup_zscore_60d", "AMD_ret_1d", "AXP_Amex_vol_20d"], "is_new": true}, {"model_id": "new_h10_CALM_GradientBoosting_N12_t2", "algo": "GradientBoosting", "regime": "CALM", "horizon": 10, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "DAX_Germany_zscore_60d", "CLX_Clorox_vol_20d", "BDX_Becton_Dickinson_ret_20d", "JNJ_ret_1d", "Nikkei_Japan_zscore_60d", "XLV_Health_zscore_60d", "ASX_Australia_ret_5d", "Core_PCE_zscore_60d", "PAYX_Paychex_ret_20d", "CPB_CampbellSoup_vol_20d"], "is_new": true}, {"model_id": "new_h10_CALM_GradientBoosting_N12_t3", "algo": "GradientBoosting", "regime": "CALM", "horizon": 10, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "HD_zscore_60d", "Core_PCE_zscore_60d", "CPB_CampbellSoup_ret_20d", "EQIX_Equinix_ret_5d", "Industrial_Production_zscore_60d", "Brent_Oil_FRED_ret_20d", "PAYX_Paychex_vol_20d", "EWJ_Japan_vol_20d", "SPY_zscore_60d", "heston_var_ev_h7"], "is_new": true}, {"model_id": "new_h10_CALM_GradientBoosting_N12_t4", "algo": "GradientBoosting", "regime": "CALM", "horizon": 10, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "AORD_AUS_zscore_60d", "spx_momentum_3d", "ES_Evergy_ret_1d", "DHR_vol_20d", "SBUX_ret_5d", "TXN_vol_20d", "EWM_Malaysia_ret_1d", "vix_acceleration_1d", "CPB_CampbellSoup_vol_20d", "DAX_Germany_vol_20d"], "is_new": true}, {"model_id": "new_h10_CALM_GradientBoosting_N12_t5", "algo": "GradientBoosting", "regime": "CALM", "horizon": 10, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "LUV_SouthwestAir_ret_5d", "XLB_Materials_zscore_60d", "EWA_Australia_zscore_60d", "US3M_Rate_vol_20d", "EOG_EOGResources_ret_5d", "spx_abs_ret_max_5d", "BDX_Becton_Dickinson_ret_20d", "US6M_Rate_ret_20d", "EWM_Malaysia_ret_1d", "EMR_Emerson_ret_20d"], "is_new": true}, {"model_id": "new_h10_CALM_GradientBoosting_N12_t6", "algo": "GradientBoosting", "regime": "CALM", "horizon": 10, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "ENB_EnbridgeInc_ret_1d", "vix_acceleration_1d", "CPB_CampbellSoup_zscore_60d", "DE_Deere_vol_20d", "CPB_CampbellSoup_vol_20d", "SPY_zscore_60d", "PAYX_Paychex_zscore_60d", "T_ret_1d", "ITT_ITTInc_ret_5d", "HD_zscore_60d"], "is_new": true}, {"model_id": "new_h10_CALM_GradientBoosting_N12_t7", "algo": "GradientBoosting", "regime": "CALM", "horizon": 10, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EOG_EOGResources_vol_20d", "AMT_AmericanTower_ret_1d", "VOD_Vodafone_zscore_60d", "HangSeng_HK_ret_1d", "IWM_SmallCap_vol_20d", "JNJ_ret_1d", "PCAR_PaccarInc_ret_5d", "EWY_Korea_zscore_60d", "NWL_Newell_ret_20d", "EWC_Canada_zscore_60d"], "is_new": true}, {"model_id": "new_h10_CALM_GradientBoosting_N15_t0", "algo": "GradientBoosting", "regime": "CALM", "horizon": 10, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWM_Malaysia_ret_1d", "CTAS_Cintas_vol_20d", "NWL_Newell_ret_20d", "US1Y_Rate_ret_5d", "DIS_vol_20d", "AORD_AUS_zscore_60d", "PFE_ret_1d", "ORCL_zscore_60d", "T10Y2Y_Spread_ret_5d", "MO_AltriaMG_ret_1d", "ENB_EnbridgeInc_ret_1d", "PAYX_Paychex_zscore_60d", "IYR_US_REIT2_zscore_60d"], "is_new": true}, {"model_id": "new_h10_CALM_GradientBoosting_N15_t1", "algo": "GradientBoosting", "regime": "CALM", "horizon": 10, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "heston_var_ev_h5", "CPB_CampbellSoup_ret_5d", "EWA_Australia_zscore_60d", "HangSeng_HK_ret_1d", "Core_PCE_zscore_60d", "XLF_Fin_vol_20d", "ENB_EnbridgeInc_ret_1d", "PFE_ret_1d", "CPB_CampbellSoup_zscore_60d", "EWA_Australia_ret_1d", "EWL_Switzerland_zscore_60d", "US3M_Rate_vol_20d", "PCAR_PaccarInc_ret_5d"], "is_new": true}, {"model_id": "new_h10_CALM_GradientBoosting_N15_t2", "algo": "GradientBoosting", "regime": "CALM", "horizon": 10, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EMR_Emerson_ret_20d", "Core_PCE_zscore_60d", "HD_zscore_60d", "LOW_Lowes_ret_5d", "EWQ_France_ret_20d", "CPB_CampbellSoup_zscore_60d", "CMCSA_ret_1d", "MO_AltriaMG_ret_1d", "US3M_Rate_zscore_60d", "QQQ_vol_20d", "HangSeng_HK_ret_1d", "XLV_Health_zscore_60d", "PPL_PPL_ret_1d"], "is_new": true}, {"model_id": "new_h10_CALM_GradientBoosting_N15_t3", "algo": "GradientBoosting", "regime": "CALM", "horizon": 10, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWC_Canada_zscore_60d", "XLK_Tech_zscore_60d", "CPB_CampbellSoup_vol_20d", "EWJ_Japan_vol_20d", "ES_Evergy_ret_1d", "M_Macys_vol_20d", "Brent_Oil_FRED_ret_20d", "HD_ret_5d", "LUV_SouthwestAir_ret_5d", "Industrial_Production_zscore_60d", "BDX_Becton_Dickinson_ret_20d", "WTI_Oil_FRED_zscore_60d", "EOG_EOGResources_ret_5d"], "is_new": true}, {"model_id": "new_h10_CALM_GradientBoosting_N15_t4", "algo": "GradientBoosting", "regime": "CALM", "horizon": 10, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "HangSeng_HK_ret_5d", "LMT_LockheedMartin_ret_1d", "Brent_Oil_FRED_ret_5d", "DIS_vol_20d", "US3M_Rate_vol_20d", "CPB_CampbellSoup_ret_20d", "hmm_p_stress", "XOM_ret_1d", "ITT_ITTInc_ret_5d", "TM_Telephone_vol_20d", "heston_var_ev_h3", "PPL_PPL_ret_1d", "AXP_Amex_ret_20d"], "is_new": true}, {"model_id": "new_h10_CALM_GradientBoosting_N15_t5", "algo": "GradientBoosting", "regime": "CALM", "horizon": 10, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWH_HongKong_ret_5d", "MSTR_Bitcoin3_ret_1d", "heston_var_ev_h5", "ES_Evergy_ret_1d", "PAYX_Paychex_ret_20d", "GE_ret_1d", "EQIX_Equinix_ret_5d", "DHR_ret_1d", "ITT_ITTInc_ret_5d", "US30Y_Rate_ret_20d", "INTC_ret_1d", "EWL_Switzerland_vol_20d", "US5Y_Rate_ret_5d"], "is_new": true}, {"model_id": "new_h10_CALM_GradientBoosting_N15_t6", "algo": "GradientBoosting", "regime": "CALM", "horizon": 10, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EFFR_ret_1d", "US6M_Rate_ret_20d", "US5Y_Rate_ret_5d", "TED_Spread_zscore_60d", "EWL_Switzerland_vol_20d", "vix_acceleration_1d", "Industrial_Production_zscore_60d", "IYM_BasicMaterials_ret_20d", "Retail_Sales_zscore_60d", "3M_ret_5d", "gjr_condvar_h1", "vix_mean_abs_ret_5d", "PAYX_Paychex_vol_20d"], "is_new": true}, {"model_id": "new_h10_CALM_GradientBoosting_N15_t7", "algo": "GradientBoosting", "regime": "CALM", "horizon": 10, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "DHR_ret_1d", "TED_Spread_vol_20d", "M_Macys_vol_20d", "US1Y_Rate_ret_5d", "EWH_HongKong_ret_5d", "TXN_vol_20d", "EXC_Exelon_zscore_60d", "EWG_Germany_ret_20d", "EFFR_vol_20d", "GILD_Gilead_ret_20d", "Retail_Sales_zscore_60d", "heston_var_ev_h5", "NWL_Newell_ret_20d"], "is_new": true}, {"model_id": "new_h10_CALM_GradientBoosting_N20_t0", "algo": "GradientBoosting", "regime": "CALM", "horizon": 10, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "FedFunds_zscore_60d", "EWL_Switzerland_zscore_60d", "SLB_Schlumberger_ret_5d", "EWQ_France_ret_20d", "MRK_Merck_zscore_60d", "AMZN_ret_5d", "NEE_NextEra_ret_20d", "TED_Spread_vol_20d", "US3M_Rate_zscore_60d", "Retail_Sales_zscore_60d", "spx_abs_ret_max_5d", "HD_ret_1d", "HUM_Humana_ret_5d", "EWY_Korea_zscore_60d", "NWL_Newell_ret_20d", "US30Y_Rate_ret_20d", "vix_mean_abs_ret_5d", "PFE_ret_1d"], "is_new": true}, {"model_id": "new_h10_CALM_GradientBoosting_N20_t1", "algo": "GradientBoosting", "regime": "CALM", "horizon": 10, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "XLV_Health_zscore_60d", "vix_mean_abs_ret_5d", "IWM_SmallCap_vol_20d", "INTC_ret_1d", "EFFR_vol_20d", "XLK_Tech_zscore_60d", "TXN_vol_20d", "3M_vol_20d", "PAYX_Paychex_zscore_60d", "CMCSA_ret_1d", "SBUX_zscore_60d", "XLY_Disc_vol_20d", "NWL_Newell_ret_20d", "LOW_Lowes_ret_5d", "EWG_Germany_vol_20d", "EQR_Equity_ret_1d", "BTI_BritishAmerican_ret_20d", "ORCL_zscore_60d"], "is_new": true}, {"model_id": "new_h10_CALM_GradientBoosting_N20_t2", "algo": "GradientBoosting", "regime": "CALM", "horizon": 10, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "SCHW_Schwab_ret_5d", "QQQ_vol_20d", "heston_var_ev_h7", "TXN_vol_20d", "NEE_NextEra_ret_20d", "vix_mean_abs_ret_5d", "HD_ret_5d", "XLB_Materials_zscore_60d", "MS_MorganStanley_ret_5d", "IBEX_Spain_ret_20d", "MSTR_Bitcoin3_ret_20d", "IWM_SmallCap_vol_20d", "T10Y2Y_Spread_ret_5d", "spx_vol_5d", "SO_SouthernCo_ret_5d", "XLV_Health_zscore_60d", "HangSeng_HK_ret_5d", "DHR_vol_20d"], "is_new": true}, {"model_id": "new_h10_CALM_GradientBoosting_N20_t3", "algo": "GradientBoosting", "regime": "CALM", "horizon": 10, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "TM_Telephone_ret_1d", "ITT_ITTInc_ret_5d", "PAYX_Paychex_ret_20d", "BLK_BlackRock_zscore_60d", "EWY_Korea_zscore_60d", "AORD_AUS_zscore_60d", "US5Y_Rate_ret_5d", "VVIX_ret_20d", "vix_acceleration_1d", "EFFR_vol_20d", "MS_MorganStanley_zscore_60d", "GILD_Gilead_ret_20d", "heston_var_ev_h3", "EWH_HongKong_ret_5d", "XLV_Health_zscore_60d", "spx_abs_ret_max_5d", "ASX_Australia_vol_20d", "DAX_Germany_zscore_60d"], "is_new": true}, {"model_id": "new_h10_CALM_GradientBoosting_N20_t4", "algo": "GradientBoosting", "regime": "CALM", "horizon": 10, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EQR_Equity_ret_1d", "LMT_LockheedMartin_ret_1d", "BTI_BritishAmerican_ret_20d", "NFCI_ret_5d", "Core_PCE_zscore_60d", "DHR_vol_20d", "US6M_Rate_ret_20d", "ITT_ITTInc_ret_5d", "HD_ret_20d", "AORD_AUS_zscore_60d", "CLX_Clorox_vol_20d", "PAYX_Paychex_zscore_60d", "XLF_Fin_vol_20d", "MO_AltriaMG_ret_1d", "LOW_Lowes_ret_5d", "US3M_Rate_zscore_60d", "T10Y2Y_Spread_ret_5d", "heston_var_ev_h3"], "is_new": true}, {"model_id": "new_h10_CALM_GradientBoosting_N20_t5", "algo": "GradientBoosting", "regime": "CALM", "horizon": 10, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "Retail_Sales_zscore_60d", "AXP_Amex_ret_20d", "EOG_EOGResources_ret_5d", "HD_ret_5d", "BTI_BritishAmerican_ret_5d", "LMT_LockheedMartin_ret_1d", "SLB_Schlumberger_ret_1d", "EWQ_France_ret_20d", "LUV_SouthwestAir_ret_5d", "XLK_Tech_zscore_60d", "NFCI_ret_5d", "EWS_Singapore_ret_5d", "EWG_Germany_ret_20d", "BLK_BlackRock_zscore_60d", "AXP_Amex_vol_20d", "EWM_Malaysia_vol_20d", "ORCL_zscore_60d", "FedFunds_zscore_60d"], "is_new": true}, {"model_id": "new_h10_CALM_GradientBoosting_N20_t6", "algo": "GradientBoosting", "regime": "CALM", "horizon": 10, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "Brent_Oil_FRED_ret_5d", "EWY_Korea_zscore_60d", "SBUX_ret_5d", "PFE_ret_1d", "T_ret_1d", "AORD_AUS_zscore_60d", "XLY_Disc_vol_20d", "MS_MorganStanley_ret_5d", "AMD_ret_5d", "ES_Evergy_ret_1d", "AXP_Amex_vol_20d", "LOW_Lowes_ret_20d", "US30Y_Rate_ret_20d", "CPB_CampbellSoup_zscore_60d", "gjr_condvar_h1", "SPY_zscore_60d", "PPL_PPL_ret_1d", "BA_ret_1d"], "is_new": true}, {"model_id": "new_h10_CALM_GradientBoosting_N20_t7", "algo": "GradientBoosting", "regime": "CALM", "horizon": 10, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "AMT_AmericanTower_ret_1d", "AMGN_Amgen_ret_1d", "DE_Deere_ret_5d", "HD_ret_1d", "EFFR_ret_1d", "Nikkei_Japan_vol_20d", "heston_ev_h3", "EWG_Germany_vol_20d", "MS_MorganStanley_zscore_60d", "EWS_Singapore_ret_5d", "EQR_Equity_ret_1d", "Brent_Oil_FRED_ret_20d", "SPY_zscore_60d", "PFE_ret_1d", "ORCL_vol_20d", "AMD_ret_1d", "MSTR_Bitcoin3_ret_20d", "VVIX_ret_20d"], "is_new": true}, {"model_id": "new_h10_CALM_GradientBoosting_N25_t0", "algo": "GradientBoosting", "regime": "CALM", "horizon": 10, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "vix_acceleration_1d", "EOG_EOGResources_ret_5d", "CPB_CampbellSoup_vol_20d", "T_ret_1d", "EWL_Switzerland_zscore_60d", "INTC_ret_5d", "EWG_Germany_vol_20d", "TM_Telephone_vol_20d", "HD_ret_1d", "Core_CPI_zscore_60d", "EOG_EOGResources_vol_20d", "ORCL_zscore_60d", "spx_abs_ret_max_5d", "DAX_Germany_vol_20d", "AMZN_ret_5d", "NOC_Northrop_ret_20d", "EWA_Australia_zscore_60d", "US30Y_Rate_ret_20d", "CPB_CampbellSoup_ret_20d", "IYM_BasicMaterials_ret_20d", "XLY_Disc_vol_20d", "SPY_zscore_60d", "3M_vol_20d"], "is_new": true}, {"model_id": "new_h10_CALM_GradientBoosting_N25_t1", "algo": "GradientBoosting", "regime": "CALM", "horizon": 10, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "QQQ_vol_20d", "SBUX_ret_5d", "AVB_AvalonBay_zscore_60d", "EOG_EOGResources_ret_5d", "SJM_JM_Smucker_ret_1d", "AXP_Amex_vol_20d", "AORD_AUS_zscore_60d", "gjr_condvar_h1", "EOG_EOGResources_vol_20d", "Core_PCE_zscore_60d", "3M_vol_20d", "heston_var_ev_h7", "JNJ_ret_1d", "T_ret_1d", "XOM_ret_20d", "US7Y_Rate_ret_20d", "XLF_Fin_vol_20d", "TED_Spread_zscore_60d", "spx_momentum_3d", "MS_MorganStanley_ret_1d", "EMR_Emerson_ret_20d", "Nikkei_Japan_vol_20d", "NEE_NextEra_ret_20d"], "is_new": true}, {"model_id": "new_h10_CALM_GradientBoosting_N25_t2", "algo": "GradientBoosting", "regime": "CALM", "horizon": 10, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "DIS_vol_20d", "TXN_vol_20d", "LOW_Lowes_ret_20d", "AMD_ret_5d", "EWG_Germany_ret_20d", "TGT_Target_zscore_60d", "TM_Telephone_vol_20d", "MO_AltriaMG_ret_1d", "EWM_Malaysia_ret_1d", "ORCL_vol_20d", "PPL_PPL_ret_1d", "Brent_Oil_FRED_ret_20d", "PCAR_PaccarInc_ret_5d", "MS_MorganStanley_ret_1d", "INTC_ret_5d", "EWH_HongKong_ret_5d", "GE_ret_1d", "QQQ_vol_20d", "EWM_Malaysia_vol_20d", "SPY_zscore_60d", "EWS_Singapore_ret_5d", "LUV_SouthwestAir_ret_5d", "EQIX_Equinix_ret_5d"], "is_new": true}, {"model_id": "new_h10_CALM_GradientBoosting_N25_t3", "algo": "GradientBoosting", "regime": "CALM", "horizon": 10, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "TED_Spread_vol_20d", "heston_var_ev_h5", "ASX_Australia_ret_5d", "spx_abs_ret_max_5d", "EXC_Exelon_zscore_60d", "PG_ret_20d", "CPB_CampbellSoup_zscore_60d", "SO_SouthernCo_ret_5d", "TM_Telephone_vol_20d", "EWM_Malaysia_vol_20d", "AMZN_ret_5d", "EWM_Malaysia_zscore_60d", "CPB_CampbellSoup_vol_20d", "spx_vol_5d", "heston_var_ev_h7", "CCI_CrownCastle_vol_20d", "ENB_EnbridgeInc_ret_1d", "hmm_p_stress", "spx_momentum_3d", "SCHW_Schwab_ret_5d", "XLY_Disc_vol_20d", "INTC_ret_1d", "HD_zscore_60d"], "is_new": true}, {"model_id": "new_h10_CALM_GradientBoosting_N25_t4", "algo": "GradientBoosting", "regime": "CALM", "horizon": 10, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "AVB_AvalonBay_zscore_60d", "MSTR_Bitcoin3_ret_20d", "ES_Evergy_ret_1d", "CI_Cigna_vol_20d", "MRK_Merck_zscore_60d", "EWY_Korea_zscore_60d", "hmm_p_stress", "MS_MorganStanley_ret_5d", "HUM_Humana_ret_5d", "Retail_Sales_zscore_60d", "XLF_Fin_vol_20d", "PLD_Prologis_ret_5d", "M_Macys_vol_20d", "CPB_CampbellSoup_zscore_60d", "TXN_vol_20d", "INTC_ret_5d", "heston_ev_h3", "HD_zscore_60d", "EWQ_France_zscore_60d", "SLB_Schlumberger_ret_1d", "ORCL_vol_20d", "NEE_NextEra_ret_20d", "DAX_Germany_zscore_60d"], "is_new": true}, {"model_id": "new_h10_CALM_GradientBoosting_N25_t5", "algo": "GradientBoosting", "regime": "CALM", "horizon": 10, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "ITT_ITTInc_ret_5d", "CLX_Clorox_vol_20d", "TXN_vol_20d", "heston_var_ev_h5", "CI_Cigna_vol_20d", "SLB_Schlumberger_ret_1d", "JNJ_ret_1d", "EWQ_France_zscore_60d", "SO_SouthernCo_ret_5d", "BTI_BritishAmerican_ret_5d", "EQIX_Equinix_ret_5d", "FedFunds_zscore_60d", "EWS_Singapore_ret_5d", "US30Y_Rate_ret_20d", "EWG_Germany_ret_20d", "vix_acceleration_1d", "CPB_CampbellSoup_vol_20d", "GE_ret_1d", "CPB_CampbellSoup_zscore_60d", "3M_ret_5d", "AVB_AvalonBay_zscore_60d", "LLY_zscore_60d", "PLD_Prologis_ret_5d"], "is_new": true}, {"model_id": "new_h10_CALM_GradientBoosting_N25_t6", "algo": "GradientBoosting", "regime": "CALM", "horizon": 10, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "XLY_Disc_vol_20d", "XLB_Materials_zscore_60d", "PPL_PPL_ret_1d", "Michigan_Sentiment_ret_20d", "INTC_ret_5d", "EWQ_France_ret_20d", "BA_ret_1d", "EXC_Exelon_zscore_60d", "TED_Spread_vol_20d", "EMR_Emerson_ret_20d", "PCAR_PaccarInc_ret_5d", "DHR_ret_1d", "CLX_Clorox_vol_20d", "Industrial_Production_zscore_60d", "LMT_LockheedMartin_ret_1d", "AXP_Amex_ret_20d", "EWA_Australia_ret_1d", "VRP_ma5", "Nikkei_Japan_zscore_60d", "SJM_JM_Smucker_ret_5d", "AMZN_ret_5d", "DAX_Germany_vol_20d", "BTI_BritishAmerican_ret_5d"], "is_new": true}, {"model_id": "new_h10_CALM_GradientBoosting_N25_t7", "algo": "GradientBoosting", "regime": "CALM", "horizon": 10, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EXC_Exelon_ret_1d", "ASX_Australia_vol_20d", "MS_MorganStanley_ret_1d", "vix_mean_abs_ret_5d", "US3M_Rate_zscore_60d", "EFFR_ret_1d", "EOG_EOGResources_ret_5d", "AVB_AvalonBay_zscore_60d", "PAYX_Paychex_zscore_60d", "EWY_Korea_ret_20d", "MRK_Merck_zscore_60d", "MO_AltriaMG_ret_1d", "MS_MorganStanley_ret_5d", "DAX_Germany_zscore_60d", "vix_acceleration_1d", "DE_Deere_ret_5d", "XLY_Disc_vol_20d", "US7Y_Rate_ret_20d", "EWY_Korea_zscore_60d", "US5Y_Rate_ret_5d", "HD_ret_20d", "XOM_ret_20d", "CPB_CampbellSoup_ret_20d"], "is_new": true}, {"model_id": "new_h10_CALM_GradientBoosting_N30_t0", "algo": "GradientBoosting", "regime": "CALM", "horizon": 10, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "SBUX_ret_5d", "XOM_ret_20d", "AVB_AvalonBay_zscore_60d", "PPL_PPL_ret_1d", "BTI_BritishAmerican_ret_20d", "SCHW_Schwab_ret_5d", "EWQ_France_zscore_60d", "AMGN_Amgen_ret_1d", "EQR_Equity_ret_1d", "CI_Cigna_vol_20d", "LOW_Lowes_ret_20d", "Michigan_Sentiment_ret_20d", "AORD_AUS_zscore_60d", "CPB_CampbellSoup_vol_20d", "EWG_Germany_vol_20d", "VOD_Vodafone_zscore_60d", "VRP_ma5", "EXC_Exelon_ret_1d", "spx_momentum_3d", "vix_acceleration_1d", "Industrial_Production_zscore_60d", "SLB_Schlumberger_ret_5d", "HD_ret_1d", "LMT_LockheedMartin_ret_1d", "PFE_ret_1d", "M_Macys_vol_20d", "EMR_Emerson_ret_20d", "BTI_BritishAmerican_ret_5d"], "is_new": true}, {"model_id": "new_h10_CALM_GradientBoosting_N30_t1", "algo": "GradientBoosting", "regime": "CALM", "horizon": 10, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "AORD_AUS_zscore_60d", "Nikkei_Japan_zscore_60d", "TM_Telephone_ret_1d", "Core_PCE_zscore_60d", "AMD_ret_5d", "NVDA_vol_20d", "VOD_Vodafone_zscore_60d", "hmm_p_stress", "CPB_CampbellSoup_vol_20d", "EWA_Australia_zscore_60d", "T_ret_1d", "PAYX_Paychex_ret_20d", "EWS_Singapore_ret_5d", "HD_ret_5d", "EWM_Malaysia_ret_1d", "ITT_ITTInc_ret_5d", "Retail_Sales_zscore_60d", "Brent_Oil_FRED_ret_5d", "CMCSA_ret_1d", "AMT_AmericanTower_ret_1d", "DAX_Germany_zscore_60d", "EWM_Malaysia_vol_20d", "IBEX_Spain_ret_20d", "ORCL_zscore_60d", "PFE_ret_1d", "AXP_Amex_vol_20d", "HD_ret_1d", "PPL_PPL_ret_1d"], "is_new": true}, {"model_id": "new_h10_CALM_GradientBoosting_N30_t2", "algo": "GradientBoosting", "regime": "CALM", "horizon": 10, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWA_Australia_zscore_60d", "SPY_zscore_60d", "TM_Telephone_vol_20d", "CPB_CampbellSoup_ret_20d", "hmm_p_stress", "EWL_Switzerland_vol_20d", "US3M_Rate_zscore_60d", "spx_abs_ret_max_5d", "PAYX_Paychex_vol_20d", "DE_Deere_vol_20d", "US3M_Rate_vol_20d", "WTI_Oil_FRED_zscore_60d", "TXN_vol_20d", "Core_CPI_zscore_60d", "US1Y_Rate_ret_5d", "SBUX_zscore_60d", "MS_MorganStanley_ret_5d", "EWY_Korea_ret_20d", "ENB_EnbridgeInc_ret_1d", "DOW_Price_zscore_60d", "CPB_CampbellSoup_ret_5d", "AXP_Amex_vol_20d", "TED_Spread_vol_20d", "CLX_Clorox_vol_20d", "HD_ret_1d", "FedFunds_zscore_60d", "AXP_Amex_ret_20d", "EWJ_Japan_vol_20d"], "is_new": true}, {"model_id": "new_h10_CALM_GradientBoosting_N30_t3", "algo": "GradientBoosting", "regime": "CALM", "horizon": 10, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "SBUX_ret_5d", "DE_Deere_ret_5d", "CCI_CrownCastle_vol_20d", "EXC_Exelon_ret_1d", "SLB_Schlumberger_ret_1d", "HD_ret_5d", "MS_MorganStanley_ret_1d", "EWQ_France_zscore_60d", "HD_zscore_60d", "MS_MorganStanley_zscore_60d", "EOG_EOGResources_vol_20d", "PAYX_Paychex_zscore_60d", "QQQ_vol_20d", "CMCSA_ret_1d", "spx_momentum_3d", "EFFR_vol_20d", "LMT_LockheedMartin_ret_1d", "HangSeng_HK_ret_5d", "MSTR_Bitcoin3_ret_5d", "EWS_Singapore_ret_5d", "EWM_Malaysia_zscore_60d", "BA_ret_1d", "SCHW_Schwab_ret_5d", "DAX_Germany_zscore_60d", "PPL_PPL_ret_1d", "EWJ_Japan_vol_20d", "BTI_BritishAmerican_ret_20d", "DE_Deere_vol_20d"], "is_new": true}, {"model_id": "new_h10_CALM_GradientBoosting_N30_t4", "algo": "GradientBoosting", "regime": "CALM", "horizon": 10, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "MSTR_Bitcoin3_ret_5d", "CPB_CampbellSoup_ret_20d", "HD_ret_5d", "MS_MorganStanley_ret_5d", "XLF_Fin_vol_20d", "LOW_Lowes_ret_20d", "spx_momentum_3d", "CTAS_Cintas_vol_20d", "EXC_Exelon_ret_1d", "ENB_EnbridgeInc_ret_1d", "heston_var_ev_h7", "NOC_Northrop_ret_20d", "BLK_BlackRock_zscore_60d", "INTC_ret_1d", "EWM_Malaysia_vol_20d", "EWA_Australia_ret_1d", "US3M_Rate_zscore_60d", "EOG_EOGResources_ret_5d", "AMD_ret_1d", "EWH_HongKong_ret_5d", "EWL_Switzerland_zscore_60d", "3M_vol_20d", "AMGN_Amgen_ret_1d", "TGT_Target_zscore_60d", "TM_Telephone_ret_1d", "EWG_Germany_vol_20d", "Michigan_Sentiment_ret_20d", "Core_CPI_zscore_60d"], "is_new": true}, {"model_id": "new_h10_CALM_GradientBoosting_N30_t5", "algo": "GradientBoosting", "regime": "CALM", "horizon": 10, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWM_Malaysia_ret_1d", "FedFunds_zscore_60d", "DAX_Germany_zscore_60d", "US1Y_Rate_ret_20d", "TED_Spread_zscore_60d", "CPB_CampbellSoup_ret_20d", "heston_var_ev_h7", "DAX_Germany_vol_20d", "SLB_Schlumberger_ret_5d", "HangSeng_HK_vol_20d", "M_Macys_vol_20d", "EQR_Equity_ret_1d", "US7Y_Rate_ret_20d", "SLB_Schlumberger_ret_1d", "EOG_EOGResources_vol_20d", "AMD_ret_5d", "CTAS_Cintas_vol_20d", "AXP_Amex_vol_20d", "SBUX_zscore_60d", "spx_vol_5d", "EQIX_Equinix_ret_5d", "TM_Telephone_ret_1d", "PLD_Prologis_ret_5d", "Core_CPI_zscore_60d", "CCI_CrownCastle_vol_20d", "NFCI_ret_5d", "HD_ret_5d", "spx_abs_ret_max_5d"], "is_new": true}, {"model_id": "new_h10_CALM_GradientBoosting_N30_t6", "algo": "GradientBoosting", "regime": "CALM", "horizon": 10, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "heston_var_ev_h7", "SBUX_vol_20d", "EWM_Malaysia_vol_20d", "EQR_Equity_ret_1d", "AORD_AUS_zscore_60d", "spx_momentum_3d", "LOW_Lowes_ret_20d", "EWH_HongKong_ret_5d", "IYM_BasicMaterials_ret_20d", "EWM_Malaysia_zscore_60d", "XLV_Health_zscore_60d", "NOC_Northrop_ret_20d", "INTC_ret_1d", "HD_ret_20d", "3M_vol_20d", "EOG_EOGResources_vol_20d", "PLD_Prologis_ret_5d", "TGT_Target_zscore_60d", "NFCI_ret_5d", "heston_var_ev_h3", "XOM_ret_20d", "AVB_AvalonBay_zscore_60d", "SCHW_Schwab_ret_5d", "GILD_Gilead_ret_20d", "Brent_Oil_FRED_ret_20d", "NVDA_vol_20d", "TM_Telephone_ret_1d", "T10Y2Y_Spread_ret_5d"], "is_new": true}, {"model_id": "new_h10_CALM_GradientBoosting_N30_t7", "algo": "GradientBoosting", "regime": "CALM", "horizon": 10, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "NEE_NextEra_ret_20d", "US3M_Rate_vol_20d", "SPY_zscore_60d", "GILD_Gilead_ret_20d", "CLX_Clorox_vol_20d", "XLY_Disc_vol_20d", "US1Y_Rate_ret_20d", "Industrial_Production_zscore_60d", "MO_AltriaMG_ret_1d", "EWC_Canada_zscore_60d", "TED_Spread_zscore_60d", "US3Y_Rate_ret_5d", "EWM_Malaysia_vol_20d", "CPB_CampbellSoup_ret_5d", "SBUX_zscore_60d", "JNJ_ret_1d", "BDX_Becton_Dickinson_ret_20d", "AMD_ret_5d", "US6M_Rate_ret_20d", "IYR_US_REIT2_zscore_60d", "vix_mean_abs_ret_5d", "CCI_CrownCastle_vol_20d", "EXC_Exelon_zscore_60d", "US30Y_Rate_ret_20d", "TGT_Target_zscore_60d", "AMT_AmericanTower_ret_1d", "Brent_Oil_FRED_ret_5d", "EWJ_Japan_vol_20d"], "is_new": true}, {"model_id": "new_h10_CALM_RandomForest_N5_t0", "algo": "RandomForest", "regime": "CALM", "horizon": 10, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "VVIX_ret_20d", "EWQ_France_zscore_60d", "TED_Spread_zscore_60d"], "is_new": true}, {"model_id": "new_h10_CALM_RandomForest_N5_t1", "algo": "RandomForest", "regime": "CALM", "horizon": 10, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWY_Korea_zscore_60d", "HangSeng_HK_ret_1d", "EWQ_France_zscore_60d"], "is_new": true}, {"model_id": "new_h10_CALM_RandomForest_N5_t2", "algo": "RandomForest", "regime": "CALM", "horizon": 10, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "heston_var_ev_h7", "NEE_NextEra_ret_20d", "SJM_JM_Smucker_ret_1d"], "is_new": true}, {"model_id": "new_h10_CALM_RandomForest_N5_t3", "algo": "RandomForest", "regime": "CALM", "horizon": 10, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "ORCL_vol_20d", "HangSeng_HK_vol_20d", "INTC_ret_5d"], "is_new": true}, {"model_id": "new_h10_CALM_RandomForest_N5_t4", "algo": "RandomForest", "regime": "CALM", "horizon": 10, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EFFR_vol_20d", "Core_PCE_zscore_60d", "heston_var_ev_h3"], "is_new": true}, {"model_id": "new_h10_CALM_RandomForest_N5_t5", "algo": "RandomForest", "regime": "CALM", "horizon": 10, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "XOM_ret_1d", "Michigan_Sentiment_ret_20d", "3M_vol_20d"], "is_new": true}, {"model_id": "new_h10_CALM_RandomForest_N5_t6", "algo": "RandomForest", "regime": "CALM", "horizon": 10, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "ITT_ITTInc_ret_5d", "Industrial_Production_zscore_60d", "heston_var_ev_h5"], "is_new": true}, {"model_id": "new_h10_CALM_RandomForest_N5_t7", "algo": "RandomForest", "regime": "CALM", "horizon": 10, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "vix_mean_abs_ret_5d", "WTI_Oil_FRED_zscore_60d", "CPB_CampbellSoup_ret_20d"], "is_new": true}, {"model_id": "new_h10_CALM_RandomForest_N8_t0", "algo": "RandomForest", "regime": "CALM", "horizon": 10, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWM_Malaysia_ret_1d", "DAX_Germany_vol_20d", "ES_Evergy_ret_1d", "BTI_BritishAmerican_ret_20d", "DOW_Price_zscore_60d", "TGT_Target_zscore_60d"], "is_new": true}, {"model_id": "new_h10_CALM_RandomForest_N8_t1", "algo": "RandomForest", "regime": "CALM", "horizon": 10, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "XLK_Tech_zscore_60d", "AXP_Amex_vol_20d", "heston_var_ev_h7", "MS_MorganStanley_zscore_60d", "VOD_Vodafone_zscore_60d", "DIS_vol_20d"], "is_new": true}, {"model_id": "new_h10_CALM_RandomForest_N8_t2", "algo": "RandomForest", "regime": "CALM", "horizon": 10, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "heston_var_ev_h3", "GE_ret_1d", "hmm_p_stress", "XLK_Tech_zscore_60d", "AMD_ret_5d", "VRP_ma5"], "is_new": true}, {"model_id": "new_h10_CALM_RandomForest_N8_t3", "algo": "RandomForest", "regime": "CALM", "horizon": 10, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "vix_acceleration_1d", "AORD_AUS_zscore_60d", "US7Y_Rate_ret_20d", "BTI_BritishAmerican_ret_5d", "EWS_Singapore_ret_5d", "DE_Deere_ret_5d"], "is_new": true}, {"model_id": "new_h10_CALM_RandomForest_N8_t4", "algo": "RandomForest", "regime": "CALM", "horizon": 10, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "DOW_Price_zscore_60d", "VOD_Vodafone_zscore_60d", "AXP_Amex_vol_20d", "EXC_Exelon_ret_1d", "PG_ret_20d", "EOG_EOGResources_ret_5d"], "is_new": true}, {"model_id": "new_h10_CALM_RandomForest_N8_t5", "algo": "RandomForest", "regime": "CALM", "horizon": 10, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "spx_abs_ret_max_5d", "CPB_CampbellSoup_vol_20d", "SLB_Schlumberger_ret_1d", "LOW_Lowes_ret_20d", "Brent_Oil_FRED_ret_20d", "EWA_Australia_zscore_60d"], "is_new": true}, {"model_id": "new_h10_CALM_RandomForest_N8_t6", "algo": "RandomForest", "regime": "CALM", "horizon": 10, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWQ_France_zscore_60d", "AORD_AUS_zscore_60d", "EWL_Switzerland_zscore_60d", "EXC_Exelon_zscore_60d", "Michigan_Sentiment_ret_20d", "Industrial_Production_zscore_60d"], "is_new": true}, {"model_id": "new_h10_CALM_RandomForest_N8_t7", "algo": "RandomForest", "regime": "CALM", "horizon": 10, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "AMZN_ret_5d", "EQIX_Equinix_ret_5d", "HD_ret_20d", "EWM_Malaysia_zscore_60d", "ORCL_vol_20d", "ASX_Australia_ret_5d"], "is_new": true}, {"model_id": "new_h10_CALM_RandomForest_N10_t0", "algo": "RandomForest", "regime": "CALM", "horizon": 10, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWS_Singapore_ret_5d", "PAYX_Paychex_ret_20d", "vix_mean_abs_ret_5d", "EWY_Korea_zscore_60d", "AVB_AvalonBay_zscore_60d", "US30Y_Rate_ret_20d", "Nikkei_Japan_vol_20d", "AXP_Amex_vol_20d"], "is_new": true}, {"model_id": "new_h10_CALM_RandomForest_N10_t1", "algo": "RandomForest", "regime": "CALM", "horizon": 10, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "DHR_ret_1d", "MS_MorganStanley_ret_1d", "NWL_Newell_ret_20d", "GD_GeneralDynamics_zscore_60d", "VVIX_ret_20d", "Retail_Sales_zscore_60d", "US1Y_Rate_ret_5d", "GE_ret_1d"], "is_new": true}, {"model_id": "new_h10_CALM_RandomForest_N10_t2", "algo": "RandomForest", "regime": "CALM", "horizon": 10, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "Core_CPI_zscore_60d", "ASX_Australia_vol_20d", "US6M_Rate_ret_20d", "DOW_Price_zscore_60d", "LUV_SouthwestAir_ret_5d", "US3M_Rate_zscore_60d", "HD_ret_1d", "HD_ret_5d"], "is_new": true}, {"model_id": "new_h10_CALM_RandomForest_N10_t3", "algo": "RandomForest", "regime": "CALM", "horizon": 10, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "GD_GeneralDynamics_zscore_60d", "T10Y2Y_Spread_ret_5d", "SPY_zscore_60d", "Brent_Oil_FRED_ret_20d", "spx_vol_5d", "EWH_HongKong_ret_5d", "HangSeng_HK_ret_1d", "CPB_CampbellSoup_ret_20d"], "is_new": true}, {"model_id": "new_h10_CALM_RandomForest_N10_t4", "algo": "RandomForest", "regime": "CALM", "horizon": 10, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "VRP_ma5", "CCI_CrownCastle_vol_20d", "EOG_EOGResources_vol_20d", "NWL_Newell_ret_20d", "US1Y_Rate_ret_5d", "GD_GeneralDynamics_zscore_60d", "SBUX_zscore_60d", "CTAS_Cintas_vol_20d"], "is_new": true}, {"model_id": "new_h10_CALM_RandomForest_N10_t5", "algo": "RandomForest", "regime": "CALM", "horizon": 10, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "ASX_Australia_vol_20d", "AVB_AvalonBay_zscore_60d", "PAYX_Paychex_zscore_60d", "XLV_Health_zscore_60d", "US30Y_Rate_ret_20d", "LMT_LockheedMartin_vol_20d", "Brent_Oil_FRED_ret_5d", "US5Y_Rate_ret_5d"], "is_new": true}, {"model_id": "new_h10_CALM_RandomForest_N10_t6", "algo": "RandomForest", "regime": "CALM", "horizon": 10, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWQ_France_zscore_60d", "US6M_Rate_ret_20d", "VRP_ma5", "ORCL_vol_20d", "US7Y_Rate_ret_20d", "EFFR_vol_20d", "gjr_condvar_h1", "SBUX_zscore_60d"], "is_new": true}, {"model_id": "new_h10_CALM_RandomForest_N10_t7", "algo": "RandomForest", "regime": "CALM", "horizon": 10, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EXC_Exelon_ret_1d", "EWL_Switzerland_vol_20d", "CTAS_Cintas_vol_20d", "Retail_Sales_zscore_60d", "MO_AltriaMG_ret_1d", "DHR_vol_20d", "EWM_Malaysia_ret_1d", "ASX_Australia_ret_5d"], "is_new": true}, {"model_id": "new_h10_CALM_RandomForest_N12_t0", "algo": "RandomForest", "regime": "CALM", "horizon": 10, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "AMD_ret_1d", "ES_Evergy_ret_1d", "GILD_Gilead_ret_20d", "SJM_JM_Smucker_ret_5d", "Nikkei_Japan_zscore_60d", "XLF_Fin_vol_20d", "MSTR_Bitcoin3_ret_20d", "NEE_NextEra_ret_20d", "EWQ_France_zscore_60d", "spx_vol_5d"], "is_new": true}, {"model_id": "new_h10_CALM_RandomForest_N12_t1", "algo": "RandomForest", "regime": "CALM", "horizon": 10, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "NVDA_vol_20d", "DHR_vol_20d", "EWM_Malaysia_zscore_60d", "ES_Evergy_ret_1d", "CLX_Clorox_vol_20d", "MS_MorganStanley_zscore_60d", "TED_Spread_zscore_60d", "EOG_EOGResources_ret_5d", "HD_ret_5d", "CPB_CampbellSoup_ret_5d"], "is_new": true}, {"model_id": "new_h10_CALM_RandomForest_N12_t2", "algo": "RandomForest", "regime": "CALM", "horizon": 10, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWJ_Japan_vol_20d", "DHR_ret_1d", "US1Y_Rate_ret_5d", "HD_ret_1d", "ASX_Australia_ret_5d", "US7Y_Rate_ret_20d", "EWL_Switzerland_zscore_60d", "XLB_Materials_zscore_60d", "EXC_Exelon_ret_1d", "DHR_vol_20d"], "is_new": true}, {"model_id": "new_h10_CALM_RandomForest_N12_t3", "algo": "RandomForest", "regime": "CALM", "horizon": 10, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWM_Malaysia_vol_20d", "EWA_Australia_ret_1d", "NWL_Newell_ret_20d", "IWM_SmallCap_vol_20d", "NEE_NextEra_ret_20d", "GILD_Gilead_ret_20d", "CLX_Clorox_vol_20d", "TM_Telephone_vol_20d", "DAX_Germany_vol_20d", "MSTR_Bitcoin3_ret_5d"], "is_new": true}, {"model_id": "new_h10_CALM_RandomForest_N12_t4", "algo": "RandomForest", "regime": "CALM", "horizon": 10, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "PAYX_Paychex_zscore_60d", "AVB_AvalonBay_zscore_60d", "SJM_JM_Smucker_ret_1d", "SLB_Schlumberger_ret_5d", "heston_ev_h3", "spx_abs_ret_max_5d", "MS_MorganStanley_ret_1d", "TGT_Target_zscore_60d", "VVIX_ret_20d", "CCI_CrownCastle_vol_20d"], "is_new": true}, {"model_id": "new_h10_CALM_RandomForest_N12_t5", "algo": "RandomForest", "regime": "CALM", "horizon": 10, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "XLB_Materials_zscore_60d", "EQR_Equity_ret_1d", "EXC_Exelon_ret_1d", "DE_Deere_vol_20d", "US6M_Rate_ret_20d", "US5Y_Rate_ret_5d", "US7Y_Rate_ret_20d", "AXP_Amex_vol_20d", "MSTR_Bitcoin3_ret_20d", "EWG_Germany_ret_20d"], "is_new": true}, {"model_id": "new_h10_CALM_RandomForest_N12_t6", "algo": "RandomForest", "regime": "CALM", "horizon": 10, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWM_Malaysia_ret_1d", "WTI_Oil_FRED_zscore_60d", "EQIX_Equinix_ret_5d", "hmm_p_stress", "spx_abs_ret_max_5d", "EQR_Equity_ret_1d", "heston_var_ev_h3", "IYR_US_REIT2_zscore_60d", "Core_PCE_zscore_60d", "DAX_Germany_vol_20d"], "is_new": true}, {"model_id": "new_h10_CALM_RandomForest_N12_t7", "algo": "RandomForest", "regime": "CALM", "horizon": 10, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "CMCSA_ret_1d", "PAYX_Paychex_ret_20d", "HD_ret_20d", "EWM_Malaysia_vol_20d", "AXP_Amex_ret_20d", "CLX_Clorox_vol_20d", "SBUX_vol_20d", "Brent_Oil_FRED_ret_20d", "US3M_Rate_zscore_60d", "VVIX_ret_20d"], "is_new": true}, {"model_id": "new_h10_CALM_RandomForest_N15_t0", "algo": "RandomForest", "regime": "CALM", "horizon": 10, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "ITT_ITTInc_ret_5d", "BTI_BritishAmerican_ret_20d", "PPL_PPL_ret_1d", "HangSeng_HK_ret_5d", "JNJ_ret_1d", "US1Y_Rate_ret_20d", "AVB_AvalonBay_zscore_60d", "XLV_Health_zscore_60d", "XLF_Fin_vol_20d", "HangSeng_HK_ret_1d", "CI_Cigna_vol_20d", "NWL_Newell_ret_20d", "HD_ret_20d"], "is_new": true}, {"model_id": "new_h10_CALM_RandomForest_N15_t1", "algo": "RandomForest", "regime": "CALM", "horizon": 10, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "AMZN_ret_5d", "JNJ_ret_1d", "LLY_zscore_60d", "US7Y_Rate_ret_20d", "EWH_HongKong_ret_5d", "Core_CPI_zscore_60d", "XLV_Health_zscore_60d", "HD_ret_5d", "DAX_Germany_zscore_60d", "NFCI_ret_5d", "BDX_Becton_Dickinson_ret_20d", "heston_ev_h3", "DIS_vol_20d"], "is_new": true}, {"model_id": "new_h10_CALM_RandomForest_N15_t2", "algo": "RandomForest", "regime": "CALM", "horizon": 10, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "spx_abs_ret_max_5d", "PLD_Prologis_ret_5d", "SO_SouthernCo_ret_5d", "LMT_LockheedMartin_vol_20d", "T_ret_1d", "XLV_Health_zscore_60d", "3M_vol_20d", "ASX_Australia_vol_20d", "Brent_Oil_FRED_ret_5d", "NVDA_vol_20d", "PAYX_Paychex_zscore_60d", "US30Y_Rate_ret_20d", "AMGN_Amgen_ret_1d"], "is_new": true}, {"model_id": "new_h10_CALM_RandomForest_N15_t3", "algo": "RandomForest", "regime": "CALM", "horizon": 10, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "heston_ev_h3", "DIS_vol_20d", "EWM_Malaysia_ret_1d", "TM_Telephone_vol_20d", "CPB_CampbellSoup_ret_20d", "ENB_EnbridgeInc_ret_1d", "PPL_PPL_ret_1d", "SCHW_Schwab_ret_5d", "GD_GeneralDynamics_zscore_60d", "AXP_Amex_vol_20d", "CTAS_Cintas_vol_20d", "MS_MorganStanley_zscore_60d", "US30Y_Rate_ret_20d"], "is_new": true}, {"model_id": "new_h10_CALM_RandomForest_N15_t4", "algo": "RandomForest", "regime": "CALM", "horizon": 10, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "IBEX_Spain_ret_20d", "TM_Telephone_ret_1d", "LOW_Lowes_ret_20d", "MSTR_Bitcoin3_ret_20d", "CCI_CrownCastle_vol_20d", "EWL_Switzerland_zscore_60d", "MS_MorganStanley_ret_1d", "MSTR_Bitcoin3_ret_5d", "EMR_Emerson_ret_20d", "NFCI_ret_5d", "EQIX_Equinix_ret_5d", "US30Y_Rate_ret_20d", "PAYX_Paychex_ret_20d"], "is_new": true}, {"model_id": "new_h10_CALM_RandomForest_N15_t5", "algo": "RandomForest", "regime": "CALM", "horizon": 10, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "MS_MorganStanley_ret_5d", "DE_Deere_ret_5d", "CLX_Clorox_vol_20d", "US3M_Rate_vol_20d", "T10Y2Y_Spread_ret_5d", "Brent_Oil_FRED_ret_5d", "CI_Cigna_vol_20d", "DOW_Price_zscore_60d", "CPB_CampbellSoup_ret_5d", "SCHW_Schwab_ret_5d", "SLB_Schlumberger_ret_1d", "Industrial_Production_zscore_60d", "VOD_Vodafone_zscore_60d"], "is_new": true}, {"model_id": "new_h10_CALM_RandomForest_N15_t6", "algo": "RandomForest", "regime": "CALM", "horizon": 10, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "Michigan_Sentiment_ret_20d", "spx_vol_5d", "AMD_ret_1d", "DIS_vol_20d", "INTC_ret_5d", "EWL_Switzerland_vol_20d", "EFFR_vol_20d", "GD_GeneralDynamics_zscore_60d", "US1Y_Rate_ret_20d", "HD_zscore_60d", "AMGN_Amgen_ret_1d", "LMT_LockheedMartin_ret_1d", "US7Y_Rate_ret_20d"], "is_new": true}, {"model_id": "new_h10_CALM_RandomForest_N15_t7", "algo": "RandomForest", "regime": "CALM", "horizon": 10, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "AORD_AUS_zscore_60d", "VRP_ma5", "DOW_Price_zscore_60d", "MS_MorganStanley_zscore_60d", "MRK_Merck_zscore_60d", "BDX_Becton_Dickinson_ret_20d", "MSTR_Bitcoin3_ret_1d", "GILD_Gilead_ret_20d", "TM_Telephone_vol_20d", "EWG_Germany_ret_20d", "SCHW_Schwab_ret_5d", "XLF_Fin_vol_20d", "CPB_CampbellSoup_ret_20d"], "is_new": true}, {"model_id": "new_h10_CALM_RandomForest_N20_t0", "algo": "RandomForest", "regime": "CALM", "horizon": 10, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "SPY_zscore_60d", "EQR_Equity_ret_1d", "NEE_NextEra_ret_20d", "CI_Cigna_vol_20d", "US30Y_Rate_ret_20d", "EWA_Australia_zscore_60d", "EMR_Emerson_ret_20d", "US5Y_Rate_ret_5d", "HD_ret_5d", "CCI_CrownCastle_vol_20d", "Brent_Oil_FRED_ret_5d", "CMCSA_ret_1d", "NFCI_ret_5d", "TED_Spread_zscore_60d", "NWL_Newell_ret_20d", "MSTR_Bitcoin3_ret_5d", "heston_var_ev_h3", "XLK_Tech_zscore_60d"], "is_new": true}, {"model_id": "new_h10_CALM_RandomForest_N20_t1", "algo": "RandomForest", "regime": "CALM", "horizon": 10, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "Brent_Oil_FRED_ret_5d", "Nikkei_Japan_zscore_60d", "SLB_Schlumberger_ret_5d", "BA_ret_1d", "HangSeng_HK_ret_1d", "AMT_AmericanTower_ret_1d", "LLY_zscore_60d", "AVB_AvalonBay_zscore_60d", "XLK_Tech_zscore_60d", "ENB_EnbridgeInc_ret_1d", "EFFR_vol_20d", "SBUX_zscore_60d", "PAYX_Paychex_zscore_60d", "EOG_EOGResources_ret_5d", "TGT_Target_zscore_60d", "MS_MorganStanley_ret_5d", "ORCL_vol_20d", "EWY_Korea_ret_20d"], "is_new": true}, {"model_id": "new_h10_CALM_RandomForest_N20_t2", "algo": "RandomForest", "regime": "CALM", "horizon": 10, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "AMD_ret_1d", "GILD_Gilead_ret_20d", "SCHW_Schwab_ret_5d", "AXP_Amex_ret_20d", "EWL_Switzerland_zscore_60d", "DAX_Germany_vol_20d", "US3Y_Rate_ret_5d", "HD_ret_20d", "CPB_CampbellSoup_zscore_60d", "MSTR_Bitcoin3_ret_20d", "HangSeng_HK_ret_5d", "GE_ret_1d", "EXC_Exelon_zscore_60d", "EWA_Australia_zscore_60d", "EWM_Malaysia_vol_20d", "IWM_SmallCap_vol_20d", "T_ret_1d", "HangSeng_HK_ret_1d"], "is_new": true}, {"model_id": "new_h10_CALM_RandomForest_N20_t3", "algo": "RandomForest", "regime": "CALM", "horizon": 10, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "BDX_Becton_Dickinson_ret_20d", "T_ret_1d", "GILD_Gilead_ret_20d", "INTC_ret_1d", "AMD_ret_5d", "EWQ_France_ret_20d", "SBUX_vol_20d", "EWC_Canada_zscore_60d", "SBUX_ret_5d", "BTI_BritishAmerican_ret_5d", "spx_momentum_3d", "DAX_Germany_zscore_60d", "IWM_SmallCap_vol_20d", "MS_MorganStanley_zscore_60d", "US5Y_Rate_ret_5d", "TXN_vol_20d", "CI_Cigna_vol_20d", "CPB_CampbellSoup_ret_5d"], "is_new": true}, {"model_id": "new_h10_CALM_RandomForest_N20_t4", "algo": "RandomForest", "regime": "CALM", "horizon": 10, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "TM_Telephone_ret_1d", "Industrial_Production_zscore_60d", "EWQ_France_ret_20d", "CPB_CampbellSoup_zscore_60d", "SLB_Schlumberger_ret_5d", "DAX_Germany_zscore_60d", "HangSeng_HK_ret_5d", "NOC_Northrop_ret_20d", "PPL_PPL_ret_1d", "BLK_BlackRock_zscore_60d", "CPB_CampbellSoup_vol_20d", "PFE_ret_1d", "SJM_JM_Smucker_ret_5d", "US30Y_Rate_ret_20d", "CPB_CampbellSoup_ret_5d", "EWM_Malaysia_zscore_60d", "LOW_Lowes_ret_20d", "XOM_ret_20d"], "is_new": true}, {"model_id": "new_h10_CALM_RandomForest_N20_t5", "algo": "RandomForest", "regime": "CALM", "horizon": 10, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "TGT_Target_zscore_60d", "Brent_Oil_FRED_ret_20d", "DE_Deere_ret_5d", "NVDA_vol_20d", "HD_zscore_60d", "CTAS_Cintas_vol_20d", "LLY_zscore_60d", "AXP_Amex_ret_20d", "PCAR_PaccarInc_ret_5d", "XLK_Tech_zscore_60d", "IYM_BasicMaterials_ret_20d", "NEE_NextEra_ret_20d", "SBUX_zscore_60d", "LOW_Lowes_ret_5d", "heston_ev_h3", "PG_ret_20d", "Nikkei_Japan_vol_20d", "US3M_Rate_zscore_60d"], "is_new": true}, {"model_id": "new_h10_CALM_RandomForest_N20_t6", "algo": "RandomForest", "regime": "CALM", "horizon": 10, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWM_Malaysia_ret_1d", "IYM_BasicMaterials_ret_20d", "SLB_Schlumberger_ret_5d", "EWL_Switzerland_vol_20d", "HD_zscore_60d", "EMR_Emerson_ret_20d", "DHR_vol_20d", "US3Y_Rate_ret_5d", "MSTR_Bitcoin3_ret_20d", "VVIX_ret_20d", "NEE_NextEra_ret_20d", "CPB_CampbellSoup_ret_20d", "VRP_ma5", "spx_abs_ret_max_5d", "ITT_ITTInc_ret_5d", "XOM_ret_20d", "EOG_EOGResources_ret_5d", "IYR_US_REIT2_zscore_60d"], "is_new": true}, {"model_id": "new_h10_CALM_RandomForest_N20_t7", "algo": "RandomForest", "regime": "CALM", "horizon": 10, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "SLB_Schlumberger_ret_1d", "CPB_CampbellSoup_ret_5d", "EWA_Australia_zscore_60d", "Nikkei_Japan_zscore_60d", "DAX_Germany_zscore_60d", "DIS_vol_20d", "ORCL_zscore_60d", "NWL_Newell_ret_20d", "PLD_Prologis_ret_5d", "US5Y_Rate_ret_5d", "EOG_EOGResources_ret_5d", "PAYX_Paychex_vol_20d", "AXP_Amex_ret_20d", "vix_mean_abs_ret_5d", "Industrial_Production_zscore_60d", "EXC_Exelon_ret_1d", "Brent_Oil_FRED_ret_20d", "EWJ_Japan_vol_20d"], "is_new": true}, {"model_id": "new_h10_CALM_RandomForest_N25_t0", "algo": "RandomForest", "regime": "CALM", "horizon": 10, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "Nikkei_Japan_vol_20d", "EFFR_ret_1d", "EWS_Singapore_ret_5d", "TED_Spread_zscore_60d", "EWQ_France_zscore_60d", "NEE_NextEra_ret_20d", "gjr_condvar_h1", "EWG_Germany_vol_20d", "hmm_p_stress", "US5Y_Rate_ret_5d", "3M_ret_5d", "JNJ_ret_1d", "heston_var_ev_h7", "SBUX_vol_20d", "EWL_Switzerland_zscore_60d", "BTI_BritishAmerican_ret_5d", "IBEX_Spain_ret_20d", "GE_ret_1d", "CPB_CampbellSoup_ret_20d", "DHR_vol_20d", "AMZN_ret_5d", "HD_zscore_60d", "ORCL_zscore_60d"], "is_new": true}, {"model_id": "new_h10_CALM_RandomForest_N25_t1", "algo": "RandomForest", "regime": "CALM", "horizon": 10, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EQR_Equity_ret_1d", "EXC_Exelon_ret_1d", "Michigan_Sentiment_ret_20d", "gjr_condvar_h1", "hmm_p_stress", "MSTR_Bitcoin3_ret_5d", "EWA_Australia_ret_1d", "DHR_vol_20d", "CI_Cigna_vol_20d", "MS_MorganStanley_ret_1d", "AMT_AmericanTower_ret_1d", "EWH_HongKong_ret_5d", "SJM_JM_Smucker_ret_5d", "3M_ret_5d", "SPY_zscore_60d", "XLY_Disc_vol_20d", "PAYX_Paychex_vol_20d", "EWL_Switzerland_vol_20d", "MS_MorganStanley_ret_5d", "ITT_ITTInc_ret_5d", "CPB_CampbellSoup_ret_5d", "heston_var_ev_h3", "XOM_ret_20d"], "is_new": true}, {"model_id": "new_h10_CALM_RandomForest_N25_t2", "algo": "RandomForest", "regime": "CALM", "horizon": 10, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "FedFunds_zscore_60d", "XLB_Materials_zscore_60d", "PPL_PPL_ret_1d", "heston_var_ev_h3", "MRK_Merck_zscore_60d", "AVB_AvalonBay_zscore_60d", "Core_PCE_zscore_60d", "HangSeng_HK_vol_20d", "TED_Spread_vol_20d", "ES_Evergy_ret_1d", "CMCSA_ret_1d", "SJM_JM_Smucker_ret_5d", "TGT_Target_zscore_60d", "hmm_p_stress", "US3M_Rate_zscore_60d", "ORCL_zscore_60d", "ASX_Australia_ret_5d", "BTI_BritishAmerican_ret_5d", "GE_ret_1d", "EWS_Singapore_ret_5d", "SCHW_Schwab_ret_5d", "EWY_Korea_ret_20d", "LOW_Lowes_ret_5d"], "is_new": true}, {"model_id": "new_h10_CALM_RandomForest_N25_t3", "algo": "RandomForest", "regime": "CALM", "horizon": 10, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "heston_ev_h3", "CPB_CampbellSoup_ret_5d", "DE_Deere_vol_20d", "EFFR_ret_1d", "Nikkei_Japan_vol_20d", "EWM_Malaysia_ret_1d", "EWM_Malaysia_zscore_60d", "AXP_Amex_vol_20d", "AMZN_ret_5d", "Brent_Oil_FRED_ret_5d", "XLB_Materials_zscore_60d", "ENB_EnbridgeInc_ret_1d", "VRP_ma5", "US30Y_Rate_ret_20d", "TED_Spread_zscore_60d", "AVB_AvalonBay_zscore_60d", "DHR_vol_20d", "US1Y_Rate_ret_20d", "SO_SouthernCo_ret_5d", "DE_Deere_ret_5d", "ASX_Australia_vol_20d", "XLK_Tech_zscore_60d", "LOW_Lowes_ret_20d"], "is_new": true}, {"model_id": "new_h10_CALM_RandomForest_N25_t4", "algo": "RandomForest", "regime": "CALM", "horizon": 10, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "XLV_Health_zscore_60d", "CMCSA_ret_1d", "SJM_JM_Smucker_ret_5d", "ORCL_zscore_60d", "MO_AltriaMG_ret_1d", "XLK_Tech_zscore_60d", "heston_ev_h3", "FedFunds_zscore_60d", "MSTR_Bitcoin3_ret_1d", "ES_Evergy_ret_1d", "NOC_Northrop_ret_20d", "Brent_Oil_FRED_ret_20d", "EWL_Switzerland_vol_20d", "ENB_EnbridgeInc_ret_1d", "DOW_Price_zscore_60d", "CPB_CampbellSoup_vol_20d", "ORCL_vol_20d", "TED_Spread_zscore_60d", "EQR_Equity_ret_1d", "TGT_Target_zscore_60d", "AMD_ret_5d", "US30Y_Rate_ret_20d", "CCI_CrownCastle_vol_20d"], "is_new": true}, {"model_id": "new_h10_CALM_RandomForest_N25_t5", "algo": "RandomForest", "regime": "CALM", "horizon": 10, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWQ_France_zscore_60d", "AMGN_Amgen_ret_1d", "3M_ret_5d", "EFFR_vol_20d", "PCAR_PaccarInc_ret_5d", "BTI_BritishAmerican_ret_5d", "EWQ_France_ret_20d", "HD_ret_5d", "ORCL_vol_20d", "DAX_Germany_zscore_60d", "IYR_US_REIT2_zscore_60d", "EWS_Singapore_ret_5d", "TM_Telephone_ret_1d", "GILD_Gilead_ret_20d", "heston_var_ev_h5", "EWL_Switzerland_zscore_60d", "Nikkei_Japan_zscore_60d", "XLF_Fin_vol_20d", "IYM_BasicMaterials_ret_20d", "DIS_vol_20d", "BDX_Becton_Dickinson_ret_20d", "EXC_Exelon_ret_1d", "CPB_CampbellSoup_zscore_60d"], "is_new": true}, {"model_id": "new_h10_CALM_RandomForest_N25_t6", "algo": "RandomForest", "regime": "CALM", "horizon": 10, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWM_Malaysia_ret_1d", "EWY_Korea_ret_20d", "ASX_Australia_ret_5d", "heston_ev_h3", "PCAR_PaccarInc_ret_5d", "MSTR_Bitcoin3_ret_20d", "XLK_Tech_zscore_60d", "LMT_LockheedMartin_vol_20d", "AXP_Amex_ret_20d", "AMGN_Amgen_ret_1d", "US30Y_Rate_ret_20d", "PLD_Prologis_ret_5d", "EOG_EOGResources_ret_5d", "EWQ_France_zscore_60d", "EWG_Germany_vol_20d", "US3M_Rate_vol_20d", "MRK_Merck_zscore_60d", "CPB_CampbellSoup_vol_20d", "QQQ_vol_20d", "TM_Telephone_ret_1d", "gjr_condvar_h1", "spx_momentum_3d", "BTI_BritishAmerican_ret_20d"], "is_new": true}, {"model_id": "new_h10_CALM_RandomForest_N25_t7", "algo": "RandomForest", "regime": "CALM", "horizon": 10, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "DOW_Price_zscore_60d", "DE_Deere_vol_20d", "TM_Telephone_ret_1d", "SJM_JM_Smucker_ret_5d", "CTAS_Cintas_vol_20d", "XLB_Materials_zscore_60d", "US1Y_Rate_ret_20d", "DE_Deere_ret_5d", "MS_MorganStanley_zscore_60d", "HD_ret_20d", "DAX_Germany_zscore_60d", "LMT_LockheedMartin_ret_1d", "TED_Spread_vol_20d", "vix_mean_abs_ret_5d", "Nikkei_Japan_vol_20d", "XLY_Disc_vol_20d", "HD_ret_1d", "GILD_Gilead_ret_20d", "XOM_ret_1d", "MS_MorganStanley_ret_5d", "EWA_Australia_ret_1d", "SLB_Schlumberger_ret_5d", "ORCL_zscore_60d"], "is_new": true}, {"model_id": "new_h10_CALM_RandomForest_N30_t0", "algo": "RandomForest", "regime": "CALM", "horizon": 10, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "IYM_BasicMaterials_ret_20d", "DAX_Germany_vol_20d", "BLK_BlackRock_zscore_60d", "AXP_Amex_ret_20d", "Core_PCE_zscore_60d", "T10Y2Y_Spread_ret_5d", "3M_ret_5d", "EWM_Malaysia_zscore_60d", "INTC_ret_1d", "Nikkei_Japan_vol_20d", "PPL_PPL_ret_1d", "DOW_Price_zscore_60d", "CLX_Clorox_vol_20d", "vix_mean_abs_ret_5d", "EQR_Equity_ret_1d", "HD_ret_1d", "AMT_AmericanTower_ret_1d", "US5Y_Rate_ret_5d", "EWM_Malaysia_ret_1d", "TGT_Target_zscore_60d", "EWQ_France_ret_20d", "LOW_Lowes_ret_20d", "DE_Deere_vol_20d", "SJM_JM_Smucker_ret_1d", "INTC_ret_5d", "Michigan_Sentiment_ret_20d", "CCI_CrownCastle_vol_20d", "PAYX_Paychex_zscore_60d"], "is_new": true}, {"model_id": "new_h10_CALM_RandomForest_N30_t1", "algo": "RandomForest", "regime": "CALM", "horizon": 10, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "SBUX_zscore_60d", "DE_Deere_ret_5d", "PG_ret_20d", "PAYX_Paychex_vol_20d", "AMD_ret_5d", "GD_GeneralDynamics_zscore_60d", "CCI_CrownCastle_vol_20d", "EQR_Equity_ret_1d", "ITT_ITTInc_ret_5d", "M_Macys_vol_20d", "EFFR_ret_1d", "FedFunds_zscore_60d", "SLB_Schlumberger_ret_5d", "PAYX_Paychex_zscore_60d", "T10Y2Y_Spread_ret_5d", "Brent_Oil_FRED_ret_20d", "VVIX_ret_20d", "EWG_Germany_ret_20d", "LOW_Lowes_ret_20d", "QQQ_vol_20d", "ENB_EnbridgeInc_ret_1d", "LMT_LockheedMartin_vol_20d", "NFCI_ret_5d", "EWS_Singapore_ret_5d", "vix_mean_abs_ret_5d", "AMD_ret_1d", "AXP_Amex_ret_20d", "SLB_Schlumberger_ret_1d"], "is_new": true}, {"model_id": "new_h10_CALM_RandomForest_N30_t2", "algo": "RandomForest", "regime": "CALM", "horizon": 10, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWA_Australia_zscore_60d", "TGT_Target_zscore_60d", "BTI_BritishAmerican_ret_20d", "VRP_ma5", "MS_MorganStanley_ret_5d", "ES_Evergy_ret_1d", "NWL_Newell_ret_20d", "EWG_Germany_ret_20d", "EWM_Malaysia_vol_20d", "Nikkei_Japan_vol_20d", "LMT_LockheedMartin_ret_1d", "EWJ_Japan_vol_20d", "T10Y2Y_Spread_ret_5d", "GILD_Gilead_ret_20d", "SBUX_vol_20d", "PG_ret_20d", "PAYX_Paychex_vol_20d", "EWQ_France_ret_20d", "EWQ_France_zscore_60d", "MSTR_Bitcoin3_ret_5d", "EMR_Emerson_ret_20d", "EWH_HongKong_ret_5d", "HD_ret_20d", "EQIX_Equinix_ret_5d", "gjr_condvar_h1", "DOW_Price_zscore_60d", "TM_Telephone_ret_1d", "heston_ev_h3"], "is_new": true}, {"model_id": "new_h10_CALM_RandomForest_N30_t3", "algo": "RandomForest", "regime": "CALM", "horizon": 10, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "PAYX_Paychex_zscore_60d", "Industrial_Production_zscore_60d", "EWL_Switzerland_vol_20d", "EFFR_ret_1d", "LLY_zscore_60d", "LOW_Lowes_ret_20d", "PFE_ret_1d", "EWG_Germany_vol_20d", "US1Y_Rate_ret_5d", "DE_Deere_ret_5d", "HD_ret_5d", "AMT_AmericanTower_ret_1d", "vix_mean_abs_ret_5d", "VRP_ma5", "EWA_Australia_zscore_60d", "MSTR_Bitcoin3_ret_1d", "IYR_US_REIT2_zscore_60d", "ES_Evergy_ret_1d", "SLB_Schlumberger_ret_1d", "VVIX_ret_20d", "US6M_Rate_ret_20d", "WTI_Oil_FRED_zscore_60d", "TM_Telephone_vol_20d", "LOW_Lowes_ret_5d", "IWM_SmallCap_vol_20d", "AVB_AvalonBay_zscore_60d", "XLY_Disc_vol_20d", "EMR_Emerson_ret_20d"], "is_new": true}, {"model_id": "new_h10_CALM_RandomForest_N30_t4", "algo": "RandomForest", "regime": "CALM", "horizon": 10, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWY_Korea_ret_20d", "SBUX_vol_20d", "BLK_BlackRock_zscore_60d", "MS_MorganStanley_ret_1d", "heston_var_ev_h7", "PCAR_PaccarInc_ret_5d", "XLY_Disc_vol_20d", "Core_CPI_zscore_60d", "AORD_AUS_zscore_60d", "GILD_Gilead_ret_20d", "DE_Deere_vol_20d", "US30Y_Rate_ret_20d", "IWM_SmallCap_vol_20d", "INTC_ret_1d", "vix_acceleration_1d", "US1Y_Rate_ret_5d", "Retail_Sales_zscore_60d", "EWG_Germany_ret_20d", "DOW_Price_zscore_60d", "Core_PCE_zscore_60d", "M_Macys_vol_20d", "WTI_Oil_FRED_zscore_60d", "XOM_ret_20d", "EQIX_Equinix_ret_5d", "EWJ_Japan_vol_20d", "heston_var_ev_h3", "TM_Telephone_ret_1d", "TGT_Target_zscore_60d"], "is_new": true}, {"model_id": "new_h10_CALM_RandomForest_N30_t5", "algo": "RandomForest", "regime": "CALM", "horizon": 10, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "MO_AltriaMG_ret_1d", "XLV_Health_zscore_60d", "US1Y_Rate_ret_20d", "US1Y_Rate_ret_5d", "PAYX_Paychex_vol_20d", "CLX_Clorox_vol_20d", "FedFunds_zscore_60d", "EOG_EOGResources_ret_5d", "DAX_Germany_vol_20d", "vix_acceleration_1d", "heston_var_ev_h5", "EWM_Malaysia_zscore_60d", "Nikkei_Japan_vol_20d", "EXC_Exelon_zscore_60d", "MS_MorganStanley_ret_1d", "MS_MorganStanley_zscore_60d", "M_Macys_vol_20d", "BDX_Becton_Dickinson_ret_20d", "HD_ret_20d", "vix_mean_abs_ret_5d", "spx_vol_5d", "EWY_Korea_zscore_60d", "IYR_US_REIT2_zscore_60d", "EWY_Korea_ret_20d", "DOW_Price_zscore_60d", "TXN_vol_20d", "GD_GeneralDynamics_zscore_60d", "EWM_Malaysia_ret_1d"], "is_new": true}, {"model_id": "new_h10_CALM_RandomForest_N30_t6", "algo": "RandomForest", "regime": "CALM", "horizon": 10, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "CTAS_Cintas_vol_20d", "MRK_Merck_zscore_60d", "HD_ret_1d", "vix_mean_abs_ret_5d", "LOW_Lowes_ret_5d", "XLF_Fin_vol_20d", "SJM_JM_Smucker_ret_1d", "3M_ret_5d", "LMT_LockheedMartin_vol_20d", "EXC_Exelon_ret_1d", "CI_Cigna_vol_20d", "EWL_Switzerland_zscore_60d", "HangSeng_HK_vol_20d", "LMT_LockheedMartin_ret_1d", "EWM_Malaysia_vol_20d", "ASX_Australia_vol_20d", "HangSeng_HK_ret_1d", "LOW_Lowes_ret_20d", "XLV_Health_zscore_60d", "EWQ_France_zscore_60d", "INTC_ret_1d", "ORCL_vol_20d", "NVDA_vol_20d", "MO_AltriaMG_ret_1d", "PCAR_PaccarInc_ret_5d", "DIS_vol_20d", "Nikkei_Japan_vol_20d", "ES_Evergy_ret_1d"], "is_new": true}, {"model_id": "new_h10_CALM_RandomForest_N30_t7", "algo": "RandomForest", "regime": "CALM", "horizon": 10, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "LOW_Lowes_ret_20d", "TM_Telephone_ret_1d", "IYR_US_REIT2_zscore_60d", "US3Y_Rate_ret_5d", "MS_MorganStanley_ret_1d", "DIS_vol_20d", "IYM_BasicMaterials_ret_20d", "MS_MorganStanley_ret_5d", "AXP_Amex_ret_20d", "SLB_Schlumberger_ret_5d", "CMCSA_ret_1d", "3M_vol_20d", "US30Y_Rate_ret_20d", "ITT_ITTInc_ret_5d", "PAYX_Paychex_ret_20d", "US1Y_Rate_ret_20d", "SJM_JM_Smucker_ret_5d", "AXP_Amex_vol_20d", "AMD_ret_1d", "HangSeng_HK_vol_20d", "LMT_LockheedMartin_vol_20d", "VOD_Vodafone_zscore_60d", "XLF_Fin_vol_20d", "heston_ev_h3", "EWC_Canada_zscore_60d", "GILD_Gilead_ret_20d", "CCI_CrownCastle_vol_20d", "MS_MorganStanley_zscore_60d"], "is_new": true}, {"model_id": "new_h10_CALM_LogisticRegression_N5_t0", "algo": "LogisticRegression", "regime": "CALM", "horizon": 10, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "GD_GeneralDynamics_zscore_60d", "EWQ_France_ret_20d", "DE_Deere_ret_5d"], "is_new": true}, {"model_id": "new_h10_CALM_LogisticRegression_N5_t1", "algo": "LogisticRegression", "regime": "CALM", "horizon": 10, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "ITT_ITTInc_ret_5d", "US30Y_Rate_ret_20d", "DE_Deere_ret_5d"], "is_new": true}, {"model_id": "new_h10_CALM_LogisticRegression_N5_t2", "algo": "LogisticRegression", "regime": "CALM", "horizon": 10, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "GD_GeneralDynamics_zscore_60d", "US3M_Rate_zscore_60d", "US30Y_Rate_ret_20d"], "is_new": true}, {"model_id": "new_h10_CALM_LogisticRegression_N5_t3", "algo": "LogisticRegression", "regime": "CALM", "horizon": 10, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "M_Macys_vol_20d", "PG_ret_20d", "vix_mean_abs_ret_5d"], "is_new": true}, {"model_id": "new_h10_CALM_LogisticRegression_N5_t4", "algo": "LogisticRegression", "regime": "CALM", "horizon": 10, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "ORCL_vol_20d", "GD_GeneralDynamics_zscore_60d", "US7Y_Rate_ret_20d"], "is_new": true}, {"model_id": "new_h10_CALM_LogisticRegression_N5_t5", "algo": "LogisticRegression", "regime": "CALM", "horizon": 10, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "T_ret_1d", "heston_var_ev_h7", "LLY_zscore_60d"], "is_new": true}, {"model_id": "new_h10_CALM_LogisticRegression_N5_t6", "algo": "LogisticRegression", "regime": "CALM", "horizon": 10, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "CI_Cigna_vol_20d", "LLY_zscore_60d", "spx_vol_5d"], "is_new": true}, {"model_id": "new_h10_CALM_LogisticRegression_N5_t7", "algo": "LogisticRegression", "regime": "CALM", "horizon": 10, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "Michigan_Sentiment_ret_20d", "MSTR_Bitcoin3_ret_5d", "QQQ_vol_20d"], "is_new": true}, {"model_id": "new_h10_CALM_LogisticRegression_N8_t0", "algo": "LogisticRegression", "regime": "CALM", "horizon": 10, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "ORCL_zscore_60d", "US3Y_Rate_ret_5d", "Brent_Oil_FRED_ret_20d", "AMGN_Amgen_ret_1d", "T10Y2Y_Spread_ret_5d", "EWY_Korea_zscore_60d"], "is_new": true}, {"model_id": "new_h10_CALM_LogisticRegression_N8_t1", "algo": "LogisticRegression", "regime": "CALM", "horizon": 10, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "SJM_JM_Smucker_ret_1d", "US3M_Rate_vol_20d", "HD_ret_20d", "MSTR_Bitcoin3_ret_1d", "MRK_Merck_zscore_60d", "ES_Evergy_ret_1d"], "is_new": true}, {"model_id": "new_h10_CALM_LogisticRegression_N8_t2", "algo": "LogisticRegression", "regime": "CALM", "horizon": 10, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "ENB_EnbridgeInc_ret_1d", "XLK_Tech_zscore_60d", "LUV_SouthwestAir_ret_5d", "EOG_EOGResources_vol_20d", "EWH_HongKong_ret_5d", "3M_ret_5d"], "is_new": true}, {"model_id": "new_h10_CALM_LogisticRegression_N8_t3", "algo": "LogisticRegression", "regime": "CALM", "horizon": 10, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "LLY_zscore_60d", "CI_Cigna_vol_20d", "EWM_Malaysia_zscore_60d", "EQIX_Equinix_ret_5d", "SCHW_Schwab_ret_5d", "AMD_ret_1d"], "is_new": true}, {"model_id": "new_h10_CALM_LogisticRegression_N8_t4", "algo": "LogisticRegression", "regime": "CALM", "horizon": 10, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "HangSeng_HK_ret_5d", "M_Macys_vol_20d", "MS_MorganStanley_zscore_60d", "EFFR_ret_1d", "ENB_EnbridgeInc_ret_1d", "EWM_Malaysia_zscore_60d"], "is_new": true}, {"model_id": "new_h10_CALM_LogisticRegression_N8_t5", "algo": "LogisticRegression", "regime": "CALM", "horizon": 10, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "SPY_zscore_60d", "US3M_Rate_zscore_60d", "EMR_Emerson_ret_20d", "IYM_BasicMaterials_ret_20d", "AXP_Amex_vol_20d", "QQQ_vol_20d"], "is_new": true}, {"model_id": "new_h10_CALM_LogisticRegression_N8_t6", "algo": "LogisticRegression", "regime": "CALM", "horizon": 10, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "VVIX_ret_20d", "Industrial_Production_zscore_60d", "WTI_Oil_FRED_zscore_60d", "PG_ret_20d", "INTC_ret_5d", "DAX_Germany_zscore_60d"], "is_new": true}, {"model_id": "new_h10_CALM_LogisticRegression_N8_t7", "algo": "LogisticRegression", "regime": "CALM", "horizon": 10, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "MO_AltriaMG_ret_1d", "US6M_Rate_ret_20d", "EWH_HongKong_ret_5d", "CCI_CrownCastle_vol_20d", "HangSeng_HK_vol_20d", "XLB_Materials_zscore_60d"], "is_new": true}, {"model_id": "new_h10_CALM_LogisticRegression_N10_t0", "algo": "LogisticRegression", "regime": "CALM", "horizon": 10, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "QQQ_vol_20d", "LOW_Lowes_ret_20d", "heston_var_ev_h7", "PLD_Prologis_ret_5d", "HD_ret_1d", "ASX_Australia_vol_20d", "XLY_Disc_vol_20d", "EWL_Switzerland_zscore_60d"], "is_new": true}, {"model_id": "new_h10_CALM_LogisticRegression_N10_t1", "algo": "LogisticRegression", "regime": "CALM", "horizon": 10, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "DHR_vol_20d", "ASX_Australia_ret_5d", "Brent_Oil_FRED_ret_5d", "CLX_Clorox_vol_20d", "VRP_ma5", "ORCL_zscore_60d", "DE_Deere_vol_20d", "INTC_ret_5d"], "is_new": true}, {"model_id": "new_h10_CALM_LogisticRegression_N10_t2", "algo": "LogisticRegression", "regime": "CALM", "horizon": 10, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "HangSeng_HK_vol_20d", "AXP_Amex_ret_20d", "NEE_NextEra_ret_20d", "EWG_Germany_vol_20d", "SO_SouthernCo_ret_5d", "NFCI_ret_5d", "NWL_Newell_ret_20d", "AMD_ret_1d"], "is_new": true}, {"model_id": "new_h10_CALM_LogisticRegression_N10_t3", "algo": "LogisticRegression", "regime": "CALM", "horizon": 10, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "AVB_AvalonBay_zscore_60d", "ORCL_zscore_60d", "NFCI_ret_5d", "QQQ_vol_20d", "AXP_Amex_vol_20d", "EWY_Korea_ret_20d", "CI_Cigna_vol_20d", "AXP_Amex_ret_20d"], "is_new": true}, {"model_id": "new_h10_CALM_LogisticRegression_N10_t4", "algo": "LogisticRegression", "regime": "CALM", "horizon": 10, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "CI_Cigna_vol_20d", "vix_acceleration_1d", "EOG_EOGResources_ret_5d", "SBUX_zscore_60d", "PCAR_PaccarInc_ret_5d", "ORCL_vol_20d", "CPB_CampbellSoup_zscore_60d", "EWG_Germany_ret_20d"], "is_new": true}, {"model_id": "new_h10_CALM_LogisticRegression_N10_t5", "algo": "LogisticRegression", "regime": "CALM", "horizon": 10, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "US7Y_Rate_ret_20d", "EWM_Malaysia_ret_1d", "heston_ev_h3", "US1Y_Rate_ret_5d", "spx_abs_ret_max_5d", "EWG_Germany_ret_20d", "vix_acceleration_1d", "LOW_Lowes_ret_20d"], "is_new": true}, {"model_id": "new_h10_CALM_LogisticRegression_N10_t6", "algo": "LogisticRegression", "regime": "CALM", "horizon": 10, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "TM_Telephone_ret_1d", "BTI_BritishAmerican_ret_20d", "DE_Deere_ret_5d", "SLB_Schlumberger_ret_1d", "GD_GeneralDynamics_zscore_60d", "vix_mean_abs_ret_5d", "IWM_SmallCap_vol_20d", "IYR_US_REIT2_zscore_60d"], "is_new": true}, {"model_id": "new_h10_CALM_LogisticRegression_N10_t7", "algo": "LogisticRegression", "regime": "CALM", "horizon": 10, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "MRK_Merck_zscore_60d", "EWM_Malaysia_ret_1d", "3M_vol_20d", "NFCI_ret_5d", "PAYX_Paychex_ret_20d", "T10Y2Y_Spread_ret_5d", "ORCL_zscore_60d", "EWL_Switzerland_zscore_60d"], "is_new": true}, {"model_id": "new_h10_CALM_LogisticRegression_N12_t0", "algo": "LogisticRegression", "regime": "CALM", "horizon": 10, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "AMZN_ret_5d", "heston_ev_h3", "ORCL_zscore_60d", "EWG_Germany_ret_20d", "spx_vol_5d", "HangSeng_HK_vol_20d", "DIS_vol_20d", "VRP_ma5", "GILD_Gilead_ret_20d", "MSTR_Bitcoin3_ret_1d"], "is_new": true}, {"model_id": "new_h10_CALM_LogisticRegression_N12_t1", "algo": "LogisticRegression", "regime": "CALM", "horizon": 10, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "Core_CPI_zscore_60d", "EWM_Malaysia_zscore_60d", "US3M_Rate_zscore_60d", "EWA_Australia_ret_1d", "TXN_vol_20d", "MS_MorganStanley_ret_5d", "EWL_Switzerland_vol_20d", "BDX_Becton_Dickinson_ret_20d", "SJM_JM_Smucker_ret_5d", "LOW_Lowes_ret_5d"], "is_new": true}, {"model_id": "new_h10_CALM_LogisticRegression_N12_t2", "algo": "LogisticRegression", "regime": "CALM", "horizon": 10, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "XLY_Disc_vol_20d", "T10Y2Y_Spread_ret_5d", "DHR_vol_20d", "Core_PCE_zscore_60d", "EMR_Emerson_ret_20d", "BTI_BritishAmerican_ret_20d", "Nikkei_Japan_vol_20d", "Core_CPI_zscore_60d", "TGT_Target_zscore_60d", "ENB_EnbridgeInc_ret_1d"], "is_new": true}, {"model_id": "new_h10_CALM_LogisticRegression_N12_t3", "algo": "LogisticRegression", "regime": "CALM", "horizon": 10, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "US3M_Rate_zscore_60d", "EXC_Exelon_ret_1d", "spx_momentum_3d", "Core_CPI_zscore_60d", "US5Y_Rate_ret_5d", "HD_ret_1d", "MSTR_Bitcoin3_ret_5d", "3M_ret_5d", "3M_vol_20d", "Retail_Sales_zscore_60d"], "is_new": true}, {"model_id": "new_h10_CALM_LogisticRegression_N12_t4", "algo": "LogisticRegression", "regime": "CALM", "horizon": 10, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "IWM_SmallCap_vol_20d", "PCAR_PaccarInc_ret_5d", "SBUX_vol_20d", "LOW_Lowes_ret_5d", "ASX_Australia_ret_5d", "GE_ret_1d", "CLX_Clorox_vol_20d", "CPB_CampbellSoup_zscore_60d", "XLY_Disc_vol_20d", "AMD_ret_5d"], "is_new": true}, {"model_id": "new_h10_CALM_LogisticRegression_N12_t5", "algo": "LogisticRegression", "regime": "CALM", "horizon": 10, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "Core_CPI_zscore_60d", "EOG_EOGResources_ret_5d", "MS_MorganStanley_zscore_60d", "XLB_Materials_zscore_60d", "DHR_ret_1d", "EWC_Canada_zscore_60d", "US5Y_Rate_ret_5d", "NWL_Newell_ret_20d", "NEE_NextEra_ret_20d", "TXN_vol_20d"], "is_new": true}, {"model_id": "new_h10_CALM_LogisticRegression_N12_t6", "algo": "LogisticRegression", "regime": "CALM", "horizon": 10, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWY_Korea_ret_20d", "DAX_Germany_vol_20d", "Retail_Sales_zscore_60d", "SO_SouthernCo_ret_5d", "ORCL_vol_20d", "TXN_vol_20d", "BDX_Becton_Dickinson_ret_20d", "T_ret_1d", "spx_momentum_3d", "SLB_Schlumberger_ret_1d"], "is_new": true}, {"model_id": "new_h10_CALM_LogisticRegression_N12_t7", "algo": "LogisticRegression", "regime": "CALM", "horizon": 10, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "AXP_Amex_vol_20d", "LMT_LockheedMartin_vol_20d", "DIS_vol_20d", "EFFR_vol_20d", "EWQ_France_ret_20d", "SO_SouthernCo_ret_5d", "US3Y_Rate_ret_5d", "HUM_Humana_ret_5d", "Brent_Oil_FRED_ret_20d", "EMR_Emerson_ret_20d"], "is_new": true}, {"model_id": "new_h10_CALM_LogisticRegression_N15_t0", "algo": "LogisticRegression", "regime": "CALM", "horizon": 10, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "PFE_ret_1d", "EWY_Korea_zscore_60d", "SBUX_ret_5d", "Michigan_Sentiment_ret_20d", "DHR_vol_20d", "Nikkei_Japan_zscore_60d", "NVDA_vol_20d", "BTI_BritishAmerican_ret_5d", "SLB_Schlumberger_ret_5d", "HangSeng_HK_ret_1d", "AXP_Amex_ret_20d", "heston_var_ev_h3", "SJM_JM_Smucker_ret_1d"], "is_new": true}, {"model_id": "new_h10_CALM_LogisticRegression_N15_t1", "algo": "LogisticRegression", "regime": "CALM", "horizon": 10, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "PFE_ret_1d", "BTI_BritishAmerican_ret_5d", "CPB_CampbellSoup_ret_20d", "MSTR_Bitcoin3_ret_20d", "Brent_Oil_FRED_ret_20d", "SBUX_zscore_60d", "EXC_Exelon_ret_1d", "Core_PCE_zscore_60d", "EWS_Singapore_ret_5d", "LUV_SouthwestAir_ret_5d", "heston_var_ev_h7", "EFFR_ret_1d", "PAYX_Paychex_vol_20d"], "is_new": true}, {"model_id": "new_h10_CALM_LogisticRegression_N15_t2", "algo": "LogisticRegression", "regime": "CALM", "horizon": 10, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWJ_Japan_vol_20d", "EQR_Equity_ret_1d", "HD_ret_20d", "3M_vol_20d", "US6M_Rate_ret_20d", "SBUX_ret_5d", "CPB_CampbellSoup_zscore_60d", "NVDA_vol_20d", "ES_Evergy_ret_1d", "TED_Spread_zscore_60d", "EWQ_France_zscore_60d", "GILD_Gilead_ret_20d", "SCHW_Schwab_ret_5d"], "is_new": true}, {"model_id": "new_h10_CALM_LogisticRegression_N15_t3", "algo": "LogisticRegression", "regime": "CALM", "horizon": 10, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "DIS_vol_20d", "XLK_Tech_zscore_60d", "CMCSA_ret_1d", "GE_ret_1d", "Nikkei_Japan_zscore_60d", "SJM_JM_Smucker_ret_1d", "ENB_EnbridgeInc_ret_1d", "NEE_NextEra_ret_20d", "AMD_ret_5d", "TM_Telephone_vol_20d", "US1Y_Rate_ret_5d", "NVDA_vol_20d", "BDX_Becton_Dickinson_ret_20d"], "is_new": true}, {"model_id": "new_h10_CALM_LogisticRegression_N15_t4", "algo": "LogisticRegression", "regime": "CALM", "horizon": 10, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "DE_Deere_vol_20d", "EQR_Equity_ret_1d", "EWH_HongKong_ret_5d", "SO_SouthernCo_ret_5d", "VVIX_ret_20d", "PAYX_Paychex_vol_20d", "ASX_Australia_vol_20d", "TXN_vol_20d", "XLY_Disc_vol_20d", "Industrial_Production_zscore_60d", "DHR_vol_20d", "AVB_AvalonBay_zscore_60d", "XOM_ret_20d"], "is_new": true}, {"model_id": "new_h10_CALM_LogisticRegression_N15_t5", "algo": "LogisticRegression", "regime": "CALM", "horizon": 10, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "XLY_Disc_vol_20d", "AXP_Amex_vol_20d", "heston_var_ev_h5", "AXP_Amex_ret_20d", "CMCSA_ret_1d", "T_ret_1d", "PAYX_Paychex_vol_20d", "CLX_Clorox_vol_20d", "BLK_BlackRock_zscore_60d", "XLB_Materials_zscore_60d", "Core_CPI_zscore_60d", "EQR_Equity_ret_1d", "heston_ev_h3"], "is_new": true}, {"model_id": "new_h10_CALM_LogisticRegression_N15_t6", "algo": "LogisticRegression", "regime": "CALM", "horizon": 10, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "TXN_vol_20d", "Core_CPI_zscore_60d", "US1Y_Rate_ret_5d", "ENB_EnbridgeInc_ret_1d", "TED_Spread_vol_20d", "EWA_Australia_ret_1d", "vix_acceleration_1d", "CLX_Clorox_vol_20d", "AMD_ret_5d", "EWG_Germany_vol_20d", "CPB_CampbellSoup_ret_5d", "US3Y_Rate_ret_5d", "VVIX_ret_20d"], "is_new": true}, {"model_id": "new_h10_CALM_LogisticRegression_N15_t7", "algo": "LogisticRegression", "regime": "CALM", "horizon": 10, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "ITT_ITTInc_ret_5d", "SBUX_ret_5d", "IBEX_Spain_ret_20d", "Core_CPI_zscore_60d", "EQR_Equity_ret_1d", "EWJ_Japan_vol_20d", "BTI_BritishAmerican_ret_20d", "IYM_BasicMaterials_ret_20d", "spx_momentum_3d", "SJM_JM_Smucker_ret_1d", "PAYX_Paychex_vol_20d", "3M_ret_5d", "SBUX_vol_20d"], "is_new": true}, {"model_id": "new_h10_CALM_LogisticRegression_N20_t0", "algo": "LogisticRegression", "regime": "CALM", "horizon": 10, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "heston_ev_h3", "XLY_Disc_vol_20d", "EQR_Equity_ret_1d", "US1Y_Rate_ret_20d", "SBUX_zscore_60d", "gjr_condvar_h1", "heston_var_ev_h5", "DHR_ret_1d", "XLV_Health_zscore_60d", "Brent_Oil_FRED_ret_20d", "EWY_Korea_ret_20d", "CTAS_Cintas_vol_20d", "AVB_AvalonBay_zscore_60d", "EWL_Switzerland_zscore_60d", "CMCSA_ret_1d", "MSTR_Bitcoin3_ret_5d", "EWA_Australia_zscore_60d", "AMZN_ret_5d"], "is_new": true}, {"model_id": "new_h10_CALM_LogisticRegression_N20_t1", "algo": "LogisticRegression", "regime": "CALM", "horizon": 10, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWG_Germany_vol_20d", "AVB_AvalonBay_zscore_60d", "VVIX_ret_20d", "XOM_ret_20d", "EWS_Singapore_ret_5d", "M_Macys_vol_20d", "HD_ret_20d", "vix_acceleration_1d", "INTC_ret_1d", "US30Y_Rate_ret_20d", "US1Y_Rate_ret_20d", "JNJ_ret_1d", "Nikkei_Japan_vol_20d", "VRP_ma5", "EWA_Australia_zscore_60d", "GILD_Gilead_ret_20d", "EXC_Exelon_zscore_60d", "ASX_Australia_vol_20d"], "is_new": true}, {"model_id": "new_h10_CALM_LogisticRegression_N20_t2", "algo": "LogisticRegression", "regime": "CALM", "horizon": 10, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "TGT_Target_zscore_60d", "WTI_Oil_FRED_zscore_60d", "EWY_Korea_ret_20d", "Core_PCE_zscore_60d", "US3M_Rate_vol_20d", "EFFR_ret_1d", "US7Y_Rate_ret_20d", "EWG_Germany_ret_20d", "CCI_CrownCastle_vol_20d", "US6M_Rate_ret_20d", "SLB_Schlumberger_ret_5d", "DE_Deere_ret_5d", "Nikkei_Japan_vol_20d", "hmm_p_stress", "EWH_HongKong_ret_5d", "AVB_AvalonBay_zscore_60d", "EWA_Australia_ret_1d", "Industrial_Production_zscore_60d"], "is_new": true}, {"model_id": "new_h10_CALM_LogisticRegression_N20_t3", "algo": "LogisticRegression", "regime": "CALM", "horizon": 10, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EFFR_ret_1d", "EWM_Malaysia_vol_20d", "ORCL_vol_20d", "CPB_CampbellSoup_ret_20d", "NWL_Newell_ret_20d", "MS_MorganStanley_ret_5d", "ASX_Australia_vol_20d", "EWG_Germany_vol_20d", "EWL_Switzerland_zscore_60d", "EWY_Korea_ret_20d", "US3M_Rate_vol_20d", "PAYX_Paychex_zscore_60d", "US6M_Rate_ret_20d", "AMD_ret_5d", "INTC_ret_1d", "TED_Spread_zscore_60d", "SJM_JM_Smucker_ret_5d", "NEE_NextEra_ret_20d"], "is_new": true}, {"model_id": "new_h10_CALM_LogisticRegression_N20_t4", "algo": "LogisticRegression", "regime": "CALM", "horizon": 10, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWG_Germany_ret_20d", "hmm_p_stress", "CPB_CampbellSoup_ret_5d", "DHR_ret_1d", "EWG_Germany_vol_20d", "HUM_Humana_ret_5d", "Brent_Oil_FRED_ret_5d", "IYM_BasicMaterials_ret_20d", "SBUX_vol_20d", "Retail_Sales_zscore_60d", "Industrial_Production_zscore_60d", "3M_vol_20d", "CTAS_Cintas_vol_20d", "EXC_Exelon_ret_1d", "heston_var_ev_h5", "GD_GeneralDynamics_zscore_60d", "DAX_Germany_vol_20d", "NFCI_ret_5d"], "is_new": true}, {"model_id": "new_h10_CALM_LogisticRegression_N20_t5", "algo": "LogisticRegression", "regime": "CALM", "horizon": 10, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "SCHW_Schwab_ret_5d", "CCI_CrownCastle_vol_20d", "ENB_EnbridgeInc_ret_1d", "EWC_Canada_zscore_60d", "EOG_EOGResources_vol_20d", "Core_PCE_zscore_60d", "ORCL_zscore_60d", "LOW_Lowes_ret_5d", "gjr_condvar_h1", "ASX_Australia_vol_20d", "US3Y_Rate_ret_5d", "DHR_ret_1d", "GE_ret_1d", "HUM_Humana_ret_5d", "SLB_Schlumberger_ret_1d", "SJM_JM_Smucker_ret_1d", "heston_var_ev_h5", "vix_mean_abs_ret_5d"], "is_new": true}, {"model_id": "new_h10_CALM_LogisticRegression_N20_t6", "algo": "LogisticRegression", "regime": "CALM", "horizon": 10, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "PFE_ret_1d", "QQQ_vol_20d", "heston_var_ev_h5", "AVB_AvalonBay_zscore_60d", "CMCSA_ret_1d", "IWM_SmallCap_vol_20d", "LUV_SouthwestAir_ret_5d", "XLY_Disc_vol_20d", "ASX_Australia_vol_20d", "BA_ret_1d", "ITT_ITTInc_ret_5d", "EWG_Germany_ret_20d", "SBUX_zscore_60d", "NWL_Newell_ret_20d", "heston_var_ev_h7", "CPB_CampbellSoup_ret_5d", "EWM_Malaysia_vol_20d", "ASX_Australia_ret_5d"], "is_new": true}, {"model_id": "new_h10_CALM_LogisticRegression_N20_t7", "algo": "LogisticRegression", "regime": "CALM", "horizon": 10, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "SBUX_zscore_60d", "vix_acceleration_1d", "DAX_Germany_zscore_60d", "AMZN_ret_5d", "SJM_JM_Smucker_ret_5d", "3M_vol_20d", "XLY_Disc_vol_20d", "MS_MorganStanley_ret_1d", "GE_ret_1d", "US5Y_Rate_ret_5d", "EMR_Emerson_ret_20d", "SBUX_ret_5d", "CPB_CampbellSoup_ret_20d", "MSTR_Bitcoin3_ret_20d", "PG_ret_20d", "EFFR_ret_1d", "XLF_Fin_vol_20d", "NVDA_vol_20d"], "is_new": true}, {"model_id": "new_h10_CALM_LogisticRegression_N25_t0", "algo": "LogisticRegression", "regime": "CALM", "horizon": 10, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "AXP_Amex_vol_20d", "US3M_Rate_vol_20d", "INTC_ret_5d", "gjr_condvar_h1", "EWH_HongKong_ret_5d", "XLK_Tech_zscore_60d", "spx_abs_ret_max_5d", "GD_GeneralDynamics_zscore_60d", "XLB_Materials_zscore_60d", "ES_Evergy_ret_1d", "EWJ_Japan_vol_20d", "3M_vol_20d", "MS_MorganStanley_ret_5d", "PPL_PPL_ret_1d", "WTI_Oil_FRED_zscore_60d", "HD_zscore_60d", "NOC_Northrop_ret_20d", "PAYX_Paychex_zscore_60d", "spx_momentum_3d", "EWG_Germany_ret_20d", "BTI_BritishAmerican_ret_20d", "MS_MorganStanley_ret_1d", "SBUX_ret_5d"], "is_new": true}, {"model_id": "new_h10_CALM_LogisticRegression_N25_t1", "algo": "LogisticRegression", "regime": "CALM", "horizon": 10, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "Core_CPI_zscore_60d", "EWM_Malaysia_zscore_60d", "EWQ_France_zscore_60d", "US5Y_Rate_ret_5d", "DE_Deere_vol_20d", "NVDA_vol_20d", "DOW_Price_zscore_60d", "T10Y2Y_Spread_ret_5d", "WTI_Oil_FRED_zscore_60d", "DAX_Germany_vol_20d", "SPY_zscore_60d", "EWC_Canada_zscore_60d", "Retail_Sales_zscore_60d", "TM_Telephone_vol_20d", "Industrial_Production_zscore_60d", "ES_Evergy_ret_1d", "MSTR_Bitcoin3_ret_20d", "HD_ret_1d", "spx_abs_ret_max_5d", "MO_AltriaMG_ret_1d", "EWY_Korea_zscore_60d", "VVIX_ret_20d", "EQR_Equity_ret_1d"], "is_new": true}, {"model_id": "new_h10_CALM_LogisticRegression_N25_t2", "algo": "LogisticRegression", "regime": "CALM", "horizon": 10, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "Brent_Oil_FRED_ret_20d", "MSTR_Bitcoin3_ret_20d", "Brent_Oil_FRED_ret_5d", "MS_MorganStanley_ret_1d", "spx_momentum_3d", "EWS_Singapore_ret_5d", "SJM_JM_Smucker_ret_5d", "SBUX_vol_20d", "ES_Evergy_ret_1d", "EWQ_France_ret_20d", "US3M_Rate_zscore_60d", "LOW_Lowes_ret_20d", "CI_Cigna_vol_20d", "SBUX_ret_5d", "GD_GeneralDynamics_zscore_60d", "INTC_ret_5d", "BTI_BritishAmerican_ret_5d", "TGT_Target_zscore_60d", "XLV_Health_zscore_60d", "TED_Spread_zscore_60d", "spx_vol_5d", "heston_ev_h3", "DIS_vol_20d"], "is_new": true}, {"model_id": "new_h10_CALM_LogisticRegression_N25_t3", "algo": "LogisticRegression", "regime": "CALM", "horizon": 10, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWH_HongKong_ret_5d", "EWY_Korea_zscore_60d", "HD_ret_5d", "DAX_Germany_vol_20d", "NVDA_vol_20d", "TM_Telephone_vol_20d", "vix_mean_abs_ret_5d", "PAYX_Paychex_ret_20d", "US30Y_Rate_ret_20d", "VRP_ma5", "DIS_vol_20d", "MS_MorganStanley_zscore_60d", "LOW_Lowes_ret_5d", "spx_vol_5d", "XLB_Materials_zscore_60d", "LUV_SouthwestAir_ret_5d", "EFFR_vol_20d", "SBUX_zscore_60d", "US7Y_Rate_ret_20d", "SBUX_vol_20d", "CTAS_Cintas_vol_20d", "EWG_Germany_ret_20d", "EOG_EOGResources_vol_20d"], "is_new": true}, {"model_id": "new_h10_CALM_LogisticRegression_N25_t4", "algo": "LogisticRegression", "regime": "CALM", "horizon": 10, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWM_Malaysia_zscore_60d", "CTAS_Cintas_vol_20d", "3M_ret_5d", "EWS_Singapore_ret_5d", "EWH_HongKong_ret_5d", "PAYX_Paychex_zscore_60d", "EWY_Korea_ret_20d", "vix_mean_abs_ret_5d", "MSTR_Bitcoin3_ret_5d", "CLX_Clorox_vol_20d", "VOD_Vodafone_zscore_60d", "Brent_Oil_FRED_ret_5d", "EWL_Switzerland_vol_20d", "IBEX_Spain_ret_20d", "heston_var_ev_h5", "Core_PCE_zscore_60d", "Nikkei_Japan_zscore_60d", "WTI_Oil_FRED_zscore_60d", "Core_CPI_zscore_60d", "Retail_Sales_zscore_60d", "XOM_ret_1d", "EWA_Australia_ret_1d", "US7Y_Rate_ret_20d"], "is_new": true}, {"model_id": "new_h10_CALM_LogisticRegression_N25_t5", "algo": "LogisticRegression", "regime": "CALM", "horizon": 10, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "ITT_ITTInc_ret_5d", "BA_ret_1d", "CPB_CampbellSoup_ret_5d", "Brent_Oil_FRED_ret_5d", "US1Y_Rate_ret_20d", "MO_AltriaMG_ret_1d", "EFFR_ret_1d", "TXN_vol_20d", "PCAR_PaccarInc_ret_5d", "SBUX_vol_20d", "MS_MorganStanley_ret_1d", "EWQ_France_ret_20d", "SJM_JM_Smucker_ret_5d", "DE_Deere_vol_20d", "IYM_BasicMaterials_ret_20d", "EWG_Germany_ret_20d", "ASX_Australia_ret_5d", "PPL_PPL_ret_1d", "PLD_Prologis_ret_5d", "NWL_Newell_ret_20d", "3M_ret_5d", "spx_vol_5d", "EWM_Malaysia_vol_20d"], "is_new": true}, {"model_id": "new_h10_CALM_LogisticRegression_N25_t6", "algo": "LogisticRegression", "regime": "CALM", "horizon": 10, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "CPB_CampbellSoup_zscore_60d", "CCI_CrownCastle_vol_20d", "XOM_ret_20d", "GILD_Gilead_ret_20d", "VVIX_ret_20d", "AMD_ret_5d", "NVDA_vol_20d", "AXP_Amex_ret_20d", "GD_GeneralDynamics_zscore_60d", "MSTR_Bitcoin3_ret_20d", "PLD_Prologis_ret_5d", "XOM_ret_1d", "EWS_Singapore_ret_5d", "EOG_EOGResources_vol_20d", "EWA_Australia_zscore_60d", "NFCI_ret_5d", "PPL_PPL_ret_1d", "AMT_AmericanTower_ret_1d", "MO_AltriaMG_ret_1d", "EWY_Korea_ret_20d", "NOC_Northrop_ret_20d", "vix_mean_abs_ret_5d", "AMGN_Amgen_ret_1d"], "is_new": true}, {"model_id": "new_h10_CALM_LogisticRegression_N25_t7", "algo": "LogisticRegression", "regime": "CALM", "horizon": 10, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWC_Canada_zscore_60d", "EXC_Exelon_zscore_60d", "XLV_Health_zscore_60d", "BLK_BlackRock_zscore_60d", "PFE_ret_1d", "DIS_vol_20d", "TED_Spread_zscore_60d", "EOG_EOGResources_ret_5d", "LMT_LockheedMartin_vol_20d", "IYR_US_REIT2_zscore_60d", "CCI_CrownCastle_vol_20d", "IBEX_Spain_ret_20d", "MSTR_Bitcoin3_ret_5d", "US3M_Rate_zscore_60d", "AXP_Amex_ret_20d", "NEE_NextEra_ret_20d", "CMCSA_ret_1d", "heston_ev_h3", "EWQ_France_ret_20d", "EWA_Australia_zscore_60d", "MS_MorganStanley_ret_5d", "Core_CPI_zscore_60d", "VVIX_ret_20d"], "is_new": true}, {"model_id": "new_h10_CALM_LogisticRegression_N30_t0", "algo": "LogisticRegression", "regime": "CALM", "horizon": 10, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "IYR_US_REIT2_zscore_60d", "EWG_Germany_ret_20d", "EWG_Germany_vol_20d", "XLK_Tech_zscore_60d", "SJM_JM_Smucker_ret_1d", "heston_var_ev_h3", "Core_CPI_zscore_60d", "US3M_Rate_vol_20d", "BLK_BlackRock_zscore_60d", "DOW_Price_zscore_60d", "DHR_ret_1d", "EWL_Switzerland_vol_20d", "vix_acceleration_1d", "EWA_Australia_ret_1d", "HD_ret_1d", "TXN_vol_20d", "FedFunds_zscore_60d", "ORCL_vol_20d", "AMGN_Amgen_ret_1d", "hmm_p_stress", "NWL_Newell_ret_20d", "AVB_AvalonBay_zscore_60d", "EWH_HongKong_ret_5d", "3M_ret_5d", "AMD_ret_1d", "TGT_Target_zscore_60d", "MS_MorganStanley_ret_1d", "EWM_Malaysia_ret_1d"], "is_new": true}, {"model_id": "new_h10_CALM_LogisticRegression_N30_t1", "algo": "LogisticRegression", "regime": "CALM", "horizon": 10, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "SJM_JM_Smucker_ret_1d", "XLV_Health_zscore_60d", "DHR_vol_20d", "FedFunds_zscore_60d", "DE_Deere_ret_5d", "heston_var_ev_h7", "TED_Spread_vol_20d", "XLF_Fin_vol_20d", "T_ret_1d", "CPB_CampbellSoup_vol_20d", "SJM_JM_Smucker_ret_5d", "MO_AltriaMG_ret_1d", "MS_MorganStanley_ret_1d", "BA_ret_1d", "XLB_Materials_zscore_60d", "heston_ev_h3", "TXN_vol_20d", "vix_acceleration_1d", "GD_GeneralDynamics_zscore_60d", "EXC_Exelon_zscore_60d", "EQIX_Equinix_ret_5d", "AMZN_ret_5d", "MS_MorganStanley_zscore_60d", "US3M_Rate_zscore_60d", "DHR_ret_1d", "LOW_Lowes_ret_5d", "MSTR_Bitcoin3_ret_5d", "LUV_SouthwestAir_ret_5d"], "is_new": true}, {"model_id": "new_h10_CALM_LogisticRegression_N30_t2", "algo": "LogisticRegression", "regime": "CALM", "horizon": 10, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "MRK_Merck_zscore_60d", "MSTR_Bitcoin3_ret_1d", "SBUX_ret_5d", "IYR_US_REIT2_zscore_60d", "CPB_CampbellSoup_zscore_60d", "EXC_Exelon_zscore_60d", "PFE_ret_1d", "EQIX_Equinix_ret_5d", "NWL_Newell_ret_20d", "SJM_JM_Smucker_ret_1d", "TXN_vol_20d", "US3M_Rate_zscore_60d", "BDX_Becton_Dickinson_ret_20d", "HD_ret_1d", "EXC_Exelon_ret_1d", "EOG_EOGResources_ret_5d", "DOW_Price_zscore_60d", "AMD_ret_5d", "ORCL_zscore_60d", "MS_MorganStanley_zscore_60d", "US30Y_Rate_ret_20d", "VRP_ma5", "EWY_Korea_zscore_60d", "SO_SouthernCo_ret_5d", "CPB_CampbellSoup_ret_20d", "AMT_AmericanTower_ret_1d", "EWA_Australia_zscore_60d", "EWJ_Japan_vol_20d"], "is_new": true}, {"model_id": "new_h10_CALM_LogisticRegression_N30_t3", "algo": "LogisticRegression", "regime": "CALM", "horizon": 10, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWA_Australia_ret_1d", "LOW_Lowes_ret_5d", "CPB_CampbellSoup_zscore_60d", "Nikkei_Japan_vol_20d", "EXC_Exelon_zscore_60d", "vix_acceleration_1d", "EXC_Exelon_ret_1d", "MS_MorganStanley_zscore_60d", "IWM_SmallCap_vol_20d", "EWL_Switzerland_zscore_60d", "EWM_Malaysia_ret_1d", "XLV_Health_zscore_60d", "VRP_ma5", "Brent_Oil_FRED_ret_20d", "heston_var_ev_h5", "AMZN_ret_5d", "EWJ_Japan_vol_20d", "PG_ret_20d", "EWG_Germany_vol_20d", "spx_vol_5d", "AMD_ret_5d", "EQIX_Equinix_ret_5d", "MRK_Merck_zscore_60d", "DAX_Germany_vol_20d", "CI_Cigna_vol_20d", "PAYX_Paychex_vol_20d", "WTI_Oil_FRED_zscore_60d", "INTC_ret_5d"], "is_new": true}, {"model_id": "new_h10_CALM_LogisticRegression_N30_t4", "algo": "LogisticRegression", "regime": "CALM", "horizon": 10, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "CCI_CrownCastle_vol_20d", "ES_Evergy_ret_1d", "IBEX_Spain_ret_20d", "XLK_Tech_zscore_60d", "DE_Deere_ret_5d", "ASX_Australia_ret_5d", "SO_SouthernCo_ret_5d", "IWM_SmallCap_vol_20d", "INTC_ret_5d", "JNJ_ret_1d", "HangSeng_HK_vol_20d", "LMT_LockheedMartin_vol_20d", "XLV_Health_zscore_60d", "LOW_Lowes_ret_5d", "AMD_ret_5d", "HUM_Humana_ret_5d", "CI_Cigna_vol_20d", "US30Y_Rate_ret_20d", "CLX_Clorox_vol_20d", "AXP_Amex_ret_20d", "SBUX_zscore_60d", "MRK_Merck_zscore_60d", "GILD_Gilead_ret_20d", "EWY_Korea_ret_20d", "TM_Telephone_ret_1d", "PLD_Prologis_ret_5d", "XOM_ret_1d", "SLB_Schlumberger_ret_1d"], "is_new": true}, {"model_id": "new_h10_CALM_LogisticRegression_N30_t5", "algo": "LogisticRegression", "regime": "CALM", "horizon": 10, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EFFR_ret_1d", "CLX_Clorox_vol_20d", "US5Y_Rate_ret_5d", "TM_Telephone_vol_20d", "AMGN_Amgen_ret_1d", "AMT_AmericanTower_ret_1d", "XLF_Fin_vol_20d", "EWM_Malaysia_ret_1d", "ORCL_zscore_60d", "heston_var_ev_h5", "SLB_Schlumberger_ret_5d", "BDX_Becton_Dickinson_ret_20d", "EWQ_France_zscore_60d", "EWS_Singapore_ret_5d", "US1Y_Rate_ret_5d", "EWM_Malaysia_zscore_60d", "QQQ_vol_20d", "DAX_Germany_vol_20d", "EWG_Germany_ret_20d", "EQR_Equity_ret_1d", "SBUX_zscore_60d", "US3M_Rate_vol_20d", "AVB_AvalonBay_zscore_60d", "EWH_HongKong_ret_5d", "XLK_Tech_zscore_60d", "ENB_EnbridgeInc_ret_1d", "EOG_EOGResources_ret_5d", "Industrial_Production_zscore_60d"], "is_new": true}, {"model_id": "new_h10_CALM_LogisticRegression_N30_t6", "algo": "LogisticRegression", "regime": "CALM", "horizon": 10, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "XLF_Fin_vol_20d", "BTI_BritishAmerican_ret_5d", "ASX_Australia_ret_5d", "AMT_AmericanTower_ret_1d", "TGT_Target_zscore_60d", "XLB_Materials_zscore_60d", "CTAS_Cintas_vol_20d", "HD_ret_5d", "LUV_SouthwestAir_ret_5d", "SLB_Schlumberger_ret_5d", "EFFR_vol_20d", "EWG_Germany_ret_20d", "AMZN_ret_5d", "T_ret_1d", "MSTR_Bitcoin3_ret_5d", "spx_momentum_3d", "XLV_Health_zscore_60d", "HangSeng_HK_ret_1d", "DOW_Price_zscore_60d", "PAYX_Paychex_vol_20d", "Retail_Sales_zscore_60d", "DIS_vol_20d", "MRK_Merck_zscore_60d", "US5Y_Rate_ret_5d", "HD_ret_1d", "SBUX_zscore_60d", "EQR_Equity_ret_1d", "Brent_Oil_FRED_ret_5d"], "is_new": true}, {"model_id": "new_h10_CALM_LogisticRegression_N30_t7", "algo": "LogisticRegression", "regime": "CALM", "horizon": 10, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "AMZN_ret_5d", "ENB_EnbridgeInc_ret_1d", "XOM_ret_20d", "DHR_vol_20d", "US3M_Rate_vol_20d", "LLY_zscore_60d", "PG_ret_20d", "DE_Deere_vol_20d", "TXN_vol_20d", "MSTR_Bitcoin3_ret_1d", "IYR_US_REIT2_zscore_60d", "gjr_condvar_h1", "XLK_Tech_zscore_60d", "vix_mean_abs_ret_5d", "EOG_EOGResources_ret_5d", "PAYX_Paychex_zscore_60d", "MS_MorganStanley_zscore_60d", "XOM_ret_1d", "Core_CPI_zscore_60d", "NEE_NextEra_ret_20d", "SLB_Schlumberger_ret_1d", "IWM_SmallCap_vol_20d", "T10Y2Y_Spread_ret_5d", "BLK_BlackRock_zscore_60d", "US30Y_Rate_ret_20d", "HD_ret_5d", "US7Y_Rate_ret_20d", "CTAS_Cintas_vol_20d"], "is_new": true}, {"model_id": "new_h10_NORMAL_XGBoost_N5_t0", "algo": "XGBoost", "regime": "NORMAL", "horizon": 10, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "MO_AltriaMG_ret_1d", "LLY_zscore_60d", "CCI_CrownCastle_vol_20d"], "is_new": true}, {"model_id": "new_h10_NORMAL_XGBoost_N5_t1", "algo": "XGBoost", "regime": "NORMAL", "horizon": 10, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "BTI_BritishAmerican_ret_20d", "EWC_Canada_zscore_60d", "EWQ_France_zscore_60d"], "is_new": true}, {"model_id": "new_h10_NORMAL_XGBoost_N5_t2", "algo": "XGBoost", "regime": "NORMAL", "horizon": 10, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "XOM_ret_20d", "PPL_PPL_ret_1d", "Brent_Oil_FRED_ret_20d"], "is_new": true}, {"model_id": "new_h10_NORMAL_XGBoost_N5_t3", "algo": "XGBoost", "regime": "NORMAL", "horizon": 10, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "INTC_ret_1d", "JNJ_ret_1d", "spx_momentum_3d"], "is_new": true}, {"model_id": "new_h10_NORMAL_XGBoost_N5_t4", "algo": "XGBoost", "regime": "NORMAL", "horizon": 10, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "US7Y_Rate_ret_20d", "US30Y_Rate_ret_20d", "CI_Cigna_vol_20d"], "is_new": true}, {"model_id": "new_h10_NORMAL_XGBoost_N5_t5", "algo": "XGBoost", "regime": "NORMAL", "horizon": 10, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "GE_ret_1d", "ENB_EnbridgeInc_ret_1d", "CMCSA_ret_1d"], "is_new": true}, {"model_id": "new_h10_NORMAL_XGBoost_N5_t6", "algo": "XGBoost", "regime": "NORMAL", "horizon": 10, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "DHR_ret_1d", "EWY_Korea_zscore_60d", "LMT_LockheedMartin_ret_1d"], "is_new": true}, {"model_id": "new_h10_NORMAL_XGBoost_N5_t7", "algo": "XGBoost", "regime": "NORMAL", "horizon": 10, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "BDX_Becton_Dickinson_ret_20d", "vix_acceleration_1d", "SCHW_Schwab_ret_5d"], "is_new": true}, {"model_id": "new_h10_NORMAL_XGBoost_N8_t0", "algo": "XGBoost", "regime": "NORMAL", "horizon": 10, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EQIX_Equinix_ret_5d", "SBUX_zscore_60d", "EFFR_ret_1d", "DE_Deere_ret_5d", "DAX_Germany_zscore_60d", "ASX_Australia_ret_5d"], "is_new": true}, {"model_id": "new_h10_NORMAL_XGBoost_N8_t1", "algo": "XGBoost", "regime": "NORMAL", "horizon": 10, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "spx_momentum_3d", "PAYX_Paychex_vol_20d", "EMR_Emerson_ret_20d", "IWM_SmallCap_vol_20d", "HD_ret_20d", "IBEX_Spain_ret_20d"], "is_new": true}, {"model_id": "new_h10_NORMAL_XGBoost_N8_t2", "algo": "XGBoost", "regime": "NORMAL", "horizon": 10, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "DE_Deere_vol_20d", "TGT_Target_zscore_60d", "BLK_BlackRock_zscore_60d", "EWJ_Japan_vol_20d", "MRK_Merck_zscore_60d", "CPB_CampbellSoup_zscore_60d"], "is_new": true}, {"model_id": "new_h10_NORMAL_XGBoost_N8_t3", "algo": "XGBoost", "regime": "NORMAL", "horizon": 10, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "HD_ret_20d", "PFE_ret_1d", "IWM_SmallCap_vol_20d", "EXC_Exelon_zscore_60d", "AMZN_ret_5d", "XLV_Health_zscore_60d"], "is_new": true}, {"model_id": "new_h10_NORMAL_XGBoost_N8_t4", "algo": "XGBoost", "regime": "NORMAL", "horizon": 10, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "ES_Evergy_ret_1d", "ORCL_zscore_60d", "heston_var_ev_h5", "PFE_ret_1d", "MRK_Merck_zscore_60d", "XOM_ret_20d"], "is_new": true}, {"model_id": "new_h10_NORMAL_XGBoost_N8_t5", "algo": "XGBoost", "regime": "NORMAL", "horizon": 10, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "MS_MorganStanley_ret_1d", "US30Y_Rate_ret_20d", "DAX_Germany_zscore_60d", "AORD_AUS_zscore_60d", "PAYX_Paychex_ret_20d", "EFFR_vol_20d"], "is_new": true}, {"model_id": "new_h10_NORMAL_XGBoost_N8_t6", "algo": "XGBoost", "regime": "NORMAL", "horizon": 10, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWJ_Japan_vol_20d", "EWY_Korea_zscore_60d", "heston_var_ev_h5", "Nikkei_Japan_zscore_60d", "TGT_Target_zscore_60d", "EXC_Exelon_zscore_60d"], "is_new": true}, {"model_id": "new_h10_NORMAL_XGBoost_N8_t7", "algo": "XGBoost", "regime": "NORMAL", "horizon": 10, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "spx_momentum_3d", "vix_acceleration_1d", "3M_ret_5d", "HangSeng_HK_ret_1d", "DE_Deere_vol_20d", "PAYX_Paychex_zscore_60d"], "is_new": true}, {"model_id": "new_h10_NORMAL_XGBoost_N10_t0", "algo": "XGBoost", "regime": "NORMAL", "horizon": 10, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "heston_var_ev_h3", "EWM_Malaysia_ret_1d", "PG_ret_20d", "AMZN_ret_5d", "SO_SouthernCo_ret_5d", "US30Y_Rate_ret_20d", "AORD_AUS_zscore_60d", "EXC_Exelon_zscore_60d"], "is_new": true}, {"model_id": "new_h10_NORMAL_XGBoost_N10_t1", "algo": "XGBoost", "regime": "NORMAL", "horizon": 10, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "AXP_Amex_vol_20d", "Michigan_Sentiment_ret_20d", "SBUX_ret_5d", "EWM_Malaysia_vol_20d", "NFCI_ret_5d", "EWC_Canada_zscore_60d", "EOG_EOGResources_ret_5d", "HangSeng_HK_vol_20d"], "is_new": true}, {"model_id": "new_h10_NORMAL_XGBoost_N10_t2", "algo": "XGBoost", "regime": "NORMAL", "horizon": 10, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "ORCL_vol_20d", "MSTR_Bitcoin3_ret_5d", "PAYX_Paychex_vol_20d", "CTAS_Cintas_vol_20d", "FedFunds_zscore_60d", "T10Y2Y_Spread_ret_5d", "DIS_vol_20d", "ASX_Australia_ret_5d"], "is_new": true}, {"model_id": "new_h10_NORMAL_XGBoost_N10_t3", "algo": "XGBoost", "regime": "NORMAL", "horizon": 10, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "NWL_Newell_ret_20d", "EQIX_Equinix_ret_5d", "SJM_JM_Smucker_ret_5d", "INTC_ret_1d", "SBUX_zscore_60d", "SCHW_Schwab_ret_5d", "PAYX_Paychex_vol_20d", "EWC_Canada_zscore_60d"], "is_new": true}, {"model_id": "new_h10_NORMAL_XGBoost_N10_t4", "algo": "XGBoost", "regime": "NORMAL", "horizon": 10, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "Core_PCE_zscore_60d", "NFCI_ret_5d", "HD_ret_5d", "EQR_Equity_ret_1d", "SCHW_Schwab_ret_5d", "DAX_Germany_vol_20d", "NEE_NextEra_ret_20d", "LMT_LockheedMartin_ret_1d"], "is_new": true}, {"model_id": "new_h10_NORMAL_XGBoost_N10_t5", "algo": "XGBoost", "regime": "NORMAL", "horizon": 10, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "gjr_condvar_h1", "EMR_Emerson_ret_20d", "TGT_Target_zscore_60d", "vix_acceleration_1d", "EWY_Korea_zscore_60d", "HUM_Humana_ret_5d", "EWG_Germany_ret_20d", "TXN_vol_20d"], "is_new": true}, {"model_id": "new_h10_NORMAL_XGBoost_N10_t6", "algo": "XGBoost", "regime": "NORMAL", "horizon": 10, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "SO_SouthernCo_ret_5d", "EWM_Malaysia_zscore_60d", "Brent_Oil_FRED_ret_20d", "PPL_PPL_ret_1d", "GILD_Gilead_ret_20d", "TXN_vol_20d", "EMR_Emerson_ret_20d", "SCHW_Schwab_ret_5d"], "is_new": true}, {"model_id": "new_h10_NORMAL_XGBoost_N10_t7", "algo": "XGBoost", "regime": "NORMAL", "horizon": 10, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "HUM_Humana_ret_5d", "XOM_ret_1d", "TXN_vol_20d", "DE_Deere_ret_5d", "3M_vol_20d", "DIS_vol_20d", "EQR_Equity_ret_1d", "JNJ_ret_1d"], "is_new": true}, {"model_id": "new_h10_NORMAL_XGBoost_N12_t0", "algo": "XGBoost", "regime": "NORMAL", "horizon": 10, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "NEE_NextEra_ret_20d", "LOW_Lowes_ret_20d", "CPB_CampbellSoup_vol_20d", "TGT_Target_zscore_60d", "MRK_Merck_zscore_60d", "Brent_Oil_FRED_ret_5d", "NOC_Northrop_ret_20d", "MSTR_Bitcoin3_ret_20d", "EWC_Canada_zscore_60d", "EWY_Korea_zscore_60d"], "is_new": true}, {"model_id": "new_h10_NORMAL_XGBoost_N12_t1", "algo": "XGBoost", "regime": "NORMAL", "horizon": 10, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EXC_Exelon_zscore_60d", "EWA_Australia_zscore_60d", "US7Y_Rate_ret_20d", "HD_ret_20d", "Brent_Oil_FRED_ret_20d", "SBUX_vol_20d", "3M_vol_20d", "Industrial_Production_zscore_60d", "EQR_Equity_ret_1d", "TED_Spread_zscore_60d"], "is_new": true}, {"model_id": "new_h10_NORMAL_XGBoost_N12_t2", "algo": "XGBoost", "regime": "NORMAL", "horizon": 10, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "GE_ret_1d", "MSTR_Bitcoin3_ret_5d", "US1Y_Rate_ret_20d", "EMR_Emerson_ret_20d", "BTI_BritishAmerican_ret_5d", "DHR_vol_20d", "HD_ret_20d", "heston_var_ev_h3", "spx_abs_ret_max_5d", "EWG_Germany_vol_20d"], "is_new": true}, {"model_id": "new_h10_NORMAL_XGBoost_N12_t3", "algo": "XGBoost", "regime": "NORMAL", "horizon": 10, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "BTI_BritishAmerican_ret_5d", "spx_momentum_3d", "PLD_Prologis_ret_5d", "EWL_Switzerland_vol_20d", "TM_Telephone_vol_20d", "T_ret_1d", "AMZN_ret_5d", "DE_Deere_vol_20d", "spx_vol_5d", "DAX_Germany_zscore_60d"], "is_new": true}, {"model_id": "new_h10_NORMAL_XGBoost_N12_t4", "algo": "XGBoost", "regime": "NORMAL", "horizon": 10, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "DOW_Price_zscore_60d", "XLB_Materials_zscore_60d", "EMR_Emerson_ret_20d", "EWM_Malaysia_vol_20d", "vix_acceleration_1d", "ASX_Australia_vol_20d", "AVB_AvalonBay_zscore_60d", "EWY_Korea_ret_20d", "CPB_CampbellSoup_ret_20d", "XLF_Fin_vol_20d"], "is_new": true}, {"model_id": "new_h10_NORMAL_XGBoost_N12_t5", "algo": "XGBoost", "regime": "NORMAL", "horizon": 10, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "hmm_p_stress", "LUV_SouthwestAir_ret_5d", "HD_ret_20d", "DAX_Germany_zscore_60d", "BTI_BritishAmerican_ret_20d", "HangSeng_HK_ret_5d", "MSTR_Bitcoin3_ret_20d", "SCHW_Schwab_ret_5d", "Nikkei_Japan_vol_20d", "CPB_CampbellSoup_vol_20d"], "is_new": true}, {"model_id": "new_h10_NORMAL_XGBoost_N12_t6", "algo": "XGBoost", "regime": "NORMAL", "horizon": 10, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "XLY_Disc_vol_20d", "Industrial_Production_zscore_60d", "EWY_Korea_ret_20d", "XOM_ret_1d", "EWQ_France_ret_20d", "US1Y_Rate_ret_5d", "heston_var_ev_h5", "EXC_Exelon_ret_1d", "EOG_EOGResources_vol_20d", "MSTR_Bitcoin3_ret_1d"], "is_new": true}, {"model_id": "new_h10_NORMAL_XGBoost_N12_t7", "algo": "XGBoost", "regime": "NORMAL", "horizon": 10, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "XLK_Tech_zscore_60d", "CCI_CrownCastle_vol_20d", "PAYX_Paychex_vol_20d", "HangSeng_HK_ret_1d", "ES_Evergy_ret_1d", "3M_ret_5d", "PPL_PPL_ret_1d", "LUV_SouthwestAir_ret_5d", "EQIX_Equinix_ret_5d", "IWM_SmallCap_vol_20d"], "is_new": true}, {"model_id": "new_h10_NORMAL_XGBoost_N15_t0", "algo": "XGBoost", "regime": "NORMAL", "horizon": 10, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "TED_Spread_zscore_60d", "vix_mean_abs_ret_5d", "AMD_ret_1d", "EWA_Australia_zscore_60d", "BTI_BritishAmerican_ret_5d", "AMGN_Amgen_ret_1d", "NVDA_vol_20d", "ES_Evergy_ret_1d", "CMCSA_ret_1d", "EWL_Switzerland_zscore_60d", "XOM_ret_20d", "BDX_Becton_Dickinson_ret_20d", "US3M_Rate_zscore_60d"], "is_new": true}, {"model_id": "new_h10_NORMAL_XGBoost_N15_t1", "algo": "XGBoost", "regime": "NORMAL", "horizon": 10, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "ENB_EnbridgeInc_ret_1d", "VRP_ma5", "3M_vol_20d", "NOC_Northrop_ret_20d", "EWM_Malaysia_zscore_60d", "vix_mean_abs_ret_5d", "HangSeng_HK_ret_1d", "SBUX_ret_5d", "Core_PCE_zscore_60d", "spx_vol_5d", "EWQ_France_zscore_60d", "M_Macys_vol_20d", "LMT_LockheedMartin_ret_1d"], "is_new": true}, {"model_id": "new_h10_NORMAL_XGBoost_N15_t2", "algo": "XGBoost", "regime": "NORMAL", "horizon": 10, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "GILD_Gilead_ret_20d", "ES_Evergy_ret_1d", "ORCL_vol_20d", "VVIX_ret_20d", "EWC_Canada_zscore_60d", "US3M_Rate_zscore_60d", "BDX_Becton_Dickinson_ret_20d", "heston_var_ev_h5", "ORCL_zscore_60d", "BTI_BritishAmerican_ret_20d", "EWM_Malaysia_ret_1d", "DE_Deere_ret_5d", "XLB_Materials_zscore_60d"], "is_new": true}, {"model_id": "new_h10_NORMAL_XGBoost_N15_t3", "algo": "XGBoost", "regime": "NORMAL", "horizon": 10, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWM_Malaysia_vol_20d", "Industrial_Production_zscore_60d", "SBUX_vol_20d", "EWH_HongKong_ret_5d", "MSTR_Bitcoin3_ret_20d", "Core_PCE_zscore_60d", "AXP_Amex_vol_20d", "MRK_Merck_zscore_60d", "3M_ret_5d", "TM_Telephone_ret_1d", "HangSeng_HK_ret_1d", "EXC_Exelon_zscore_60d", "T10Y2Y_Spread_ret_5d"], "is_new": true}, {"model_id": "new_h10_NORMAL_XGBoost_N15_t4", "algo": "XGBoost", "regime": "NORMAL", "horizon": 10, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "CLX_Clorox_vol_20d", "ASX_Australia_vol_20d", "US1Y_Rate_ret_5d", "NOC_Northrop_ret_20d", "US1Y_Rate_ret_20d", "EWS_Singapore_ret_5d", "vix_acceleration_1d", "TM_Telephone_ret_1d", "LMT_LockheedMartin_ret_1d", "T_ret_1d", "MO_AltriaMG_ret_1d", "CPB_CampbellSoup_ret_5d", "EFFR_vol_20d"], "is_new": true}, {"model_id": "new_h10_NORMAL_XGBoost_N15_t5", "algo": "XGBoost", "regime": "NORMAL", "horizon": 10, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "AXP_Amex_ret_20d", "CLX_Clorox_vol_20d", "IYR_US_REIT2_zscore_60d", "heston_var_ev_h3", "XLF_Fin_vol_20d", "US1Y_Rate_ret_5d", "XLB_Materials_zscore_60d", "ENB_EnbridgeInc_ret_1d", "XOM_ret_20d", "US1Y_Rate_ret_20d", "TED_Spread_vol_20d", "ORCL_vol_20d", "BLK_BlackRock_zscore_60d"], "is_new": true}, {"model_id": "new_h10_NORMAL_XGBoost_N15_t6", "algo": "XGBoost", "regime": "NORMAL", "horizon": 10, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "LMT_LockheedMartin_vol_20d", "US3M_Rate_zscore_60d", "SPY_zscore_60d", "EWJ_Japan_vol_20d", "spx_abs_ret_max_5d", "HD_ret_5d", "US3Y_Rate_ret_5d", "XLY_Disc_vol_20d", "LUV_SouthwestAir_ret_5d", "M_Macys_vol_20d", "NVDA_vol_20d", "NFCI_ret_5d", "EWL_Switzerland_vol_20d"], "is_new": true}, {"model_id": "new_h10_NORMAL_XGBoost_N15_t7", "algo": "XGBoost", "regime": "NORMAL", "horizon": 10, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "TM_Telephone_vol_20d", "AXP_Amex_vol_20d", "ITT_ITTInc_ret_5d", "spx_momentum_3d", "vix_acceleration_1d", "SJM_JM_Smucker_ret_5d", "ASX_Australia_ret_5d", "heston_var_ev_h5", "EWA_Australia_zscore_60d", "PAYX_Paychex_vol_20d", "NFCI_ret_5d", "ORCL_vol_20d", "MO_AltriaMG_ret_1d"], "is_new": true}, {"model_id": "new_h10_NORMAL_XGBoost_N20_t0", "algo": "XGBoost", "regime": "NORMAL", "horizon": 10, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "HangSeng_HK_vol_20d", "EWS_Singapore_ret_5d", "EXC_Exelon_ret_1d", "US1Y_Rate_ret_5d", "EWC_Canada_zscore_60d", "T10Y2Y_Spread_ret_5d", "EWH_HongKong_ret_5d", "HD_ret_1d", "TED_Spread_vol_20d", "EWL_Switzerland_vol_20d", "3M_ret_5d", "EWM_Malaysia_ret_1d", "PAYX_Paychex_ret_20d", "CPB_CampbellSoup_zscore_60d", "EOG_EOGResources_ret_5d", "IYR_US_REIT2_zscore_60d", "US30Y_Rate_ret_20d", "HD_ret_5d"], "is_new": true}, {"model_id": "new_h10_NORMAL_XGBoost_N20_t1", "algo": "XGBoost", "regime": "NORMAL", "horizon": 10, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "Brent_Oil_FRED_ret_20d", "EWL_Switzerland_vol_20d", "SBUX_vol_20d", "HD_ret_1d", "ES_Evergy_ret_1d", "BLK_BlackRock_zscore_60d", "DAX_Germany_zscore_60d", "vix_acceleration_1d", "heston_var_ev_h7", "TM_Telephone_vol_20d", "DE_Deere_ret_5d", "SO_SouthernCo_ret_5d", "CCI_CrownCastle_vol_20d", "XOM_ret_20d", "ASX_Australia_vol_20d", "US1Y_Rate_ret_20d", "heston_ev_h3", "HangSeng_HK_vol_20d"], "is_new": true}, {"model_id": "new_h10_NORMAL_XGBoost_N20_t2", "algo": "XGBoost", "regime": "NORMAL", "horizon": 10, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWA_Australia_ret_1d", "DHR_vol_20d", "CCI_CrownCastle_vol_20d", "EWS_Singapore_ret_5d", "hmm_p_stress", "CMCSA_ret_1d", "US5Y_Rate_ret_5d", "BTI_BritishAmerican_ret_5d", "AMT_AmericanTower_ret_1d", "heston_ev_h3", "PLD_Prologis_ret_5d", "LOW_Lowes_ret_5d", "ES_Evergy_ret_1d", "SBUX_ret_5d", "US30Y_Rate_ret_20d", "XLY_Disc_vol_20d", "MS_MorganStanley_zscore_60d", "DIS_vol_20d"], "is_new": true}, {"model_id": "new_h10_NORMAL_XGBoost_N20_t3", "algo": "XGBoost", "regime": "NORMAL", "horizon": 10, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "Industrial_Production_zscore_60d", "TED_Spread_vol_20d", "VRP_ma5", "M_Macys_vol_20d", "US1Y_Rate_ret_5d", "LMT_LockheedMartin_ret_1d", "XLF_Fin_vol_20d", "EWY_Korea_zscore_60d", "EFFR_ret_1d", "CMCSA_ret_1d", "SBUX_vol_20d", "XOM_ret_20d", "EFFR_vol_20d", "NVDA_vol_20d", "ASX_Australia_vol_20d", "SO_SouthernCo_ret_5d", "CPB_CampbellSoup_ret_5d", "EWH_HongKong_ret_5d"], "is_new": true}, {"model_id": "new_h10_NORMAL_XGBoost_N20_t4", "algo": "XGBoost", "regime": "NORMAL", "horizon": 10, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "US3M_Rate_vol_20d", "EMR_Emerson_ret_20d", "EWM_Malaysia_vol_20d", "DAX_Germany_vol_20d", "AMD_ret_1d", "PFE_ret_1d", "SLB_Schlumberger_ret_1d", "HD_ret_5d", "NEE_NextEra_ret_20d", "ORCL_vol_20d", "XLB_Materials_zscore_60d", "heston_ev_h3", "JNJ_ret_1d", "MSTR_Bitcoin3_ret_20d", "CPB_CampbellSoup_vol_20d", "PAYX_Paychex_ret_20d", "CLX_Clorox_vol_20d", "TM_Telephone_ret_1d"], "is_new": true}, {"model_id": "new_h10_NORMAL_XGBoost_N20_t5", "algo": "XGBoost", "regime": "NORMAL", "horizon": 10, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWH_HongKong_ret_5d", "T_ret_1d", "US1Y_Rate_ret_5d", "DE_Deere_ret_5d", "spx_abs_ret_max_5d", "PPL_PPL_ret_1d", "US3M_Rate_vol_20d", "GILD_Gilead_ret_20d", "CI_Cigna_vol_20d", "EWS_Singapore_ret_5d", "hmm_p_stress", "spx_vol_5d", "ORCL_zscore_60d", "EWM_Malaysia_vol_20d", "Brent_Oil_FRED_ret_5d", "gjr_condvar_h1", "SJM_JM_Smucker_ret_1d", "SCHW_Schwab_ret_5d"], "is_new": true}, {"model_id": "new_h10_NORMAL_XGBoost_N20_t6", "algo": "XGBoost", "regime": "NORMAL", "horizon": 10, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "Nikkei_Japan_vol_20d", "CPB_CampbellSoup_zscore_60d", "HangSeng_HK_vol_20d", "US3Y_Rate_ret_5d", "ASX_Australia_vol_20d", "HD_ret_20d", "SLB_Schlumberger_ret_5d", "INTC_ret_1d", "TXN_vol_20d", "HD_zscore_60d", "LUV_SouthwestAir_ret_5d", "spx_vol_5d", "DE_Deere_vol_20d", "Core_CPI_zscore_60d", "BTI_BritishAmerican_ret_5d", "EFFR_vol_20d", "XLY_Disc_vol_20d", "IBEX_Spain_ret_20d"], "is_new": true}, {"model_id": "new_h10_NORMAL_XGBoost_N20_t7", "algo": "XGBoost", "regime": "NORMAL", "horizon": 10, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "AMZN_ret_5d", "BLK_BlackRock_zscore_60d", "ORCL_vol_20d", "EFFR_vol_20d", "CLX_Clorox_vol_20d", "heston_var_ev_h3", "SJM_JM_Smucker_ret_1d", "HD_ret_1d", "TGT_Target_zscore_60d", "TED_Spread_vol_20d", "CMCSA_ret_1d", "VVIX_ret_20d", "AMT_AmericanTower_ret_1d", "XLK_Tech_zscore_60d", "MSTR_Bitcoin3_ret_5d", "AORD_AUS_zscore_60d", "SBUX_zscore_60d", "MS_MorganStanley_zscore_60d"], "is_new": true}, {"model_id": "new_h10_NORMAL_XGBoost_N25_t0", "algo": "XGBoost", "regime": "NORMAL", "horizon": 10, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "NVDA_vol_20d", "LOW_Lowes_ret_20d", "BLK_BlackRock_zscore_60d", "CTAS_Cintas_vol_20d", "MS_MorganStanley_ret_1d", "EWS_Singapore_ret_5d", "T_ret_1d", "LLY_zscore_60d", "PPL_PPL_ret_1d", "EWY_Korea_ret_20d", "BTI_BritishAmerican_ret_5d", "XLK_Tech_zscore_60d", "NFCI_ret_5d", "TGT_Target_zscore_60d", "HUM_Humana_ret_5d", "PFE_ret_1d", "AMGN_Amgen_ret_1d", "US1Y_Rate_ret_20d", "CPB_CampbellSoup_vol_20d", "GE_ret_1d", "vix_acceleration_1d", "ASX_Australia_vol_20d", "SLB_Schlumberger_ret_1d"], "is_new": true}, {"model_id": "new_h10_NORMAL_XGBoost_N25_t1", "algo": "XGBoost", "regime": "NORMAL", "horizon": 10, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "SJM_JM_Smucker_ret_5d", "LOW_Lowes_ret_5d", "ENB_EnbridgeInc_ret_1d", "MSTR_Bitcoin3_ret_1d", "EWG_Germany_vol_20d", "vix_acceleration_1d", "EWC_Canada_zscore_60d", "EMR_Emerson_ret_20d", "LLY_zscore_60d", "ORCL_vol_20d", "JNJ_ret_1d", "TM_Telephone_vol_20d", "US3M_Rate_vol_20d", "US5Y_Rate_ret_5d", "EWJ_Japan_vol_20d", "Core_PCE_zscore_60d", "ORCL_zscore_60d", "EWH_HongKong_ret_5d", "ASX_Australia_ret_5d", "VRP_ma5", "3M_vol_20d", "Brent_Oil_FRED_ret_5d", "Brent_Oil_FRED_ret_20d"], "is_new": true}, {"model_id": "new_h10_NORMAL_XGBoost_N25_t2", "algo": "XGBoost", "regime": "NORMAL", "horizon": 10, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "BA_ret_1d", "EWM_Malaysia_vol_20d", "NFCI_ret_5d", "XOM_ret_1d", "CMCSA_ret_1d", "LUV_SouthwestAir_ret_5d", "JNJ_ret_1d", "GILD_Gilead_ret_20d", "DIS_vol_20d", "Nikkei_Japan_vol_20d", "BDX_Becton_Dickinson_ret_20d", "EFFR_vol_20d", "ORCL_vol_20d", "SBUX_zscore_60d", "MS_MorganStanley_zscore_60d", "BTI_BritishAmerican_ret_5d", "Industrial_Production_zscore_60d", "CLX_Clorox_vol_20d", "US30Y_Rate_ret_20d", "BTI_BritishAmerican_ret_20d", "US7Y_Rate_ret_20d", "EWS_Singapore_ret_5d", "EWY_Korea_ret_20d"], "is_new": true}, {"model_id": "new_h10_NORMAL_XGBoost_N25_t3", "algo": "XGBoost", "regime": "NORMAL", "horizon": 10, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWY_Korea_zscore_60d", "PFE_ret_1d", "GILD_Gilead_ret_20d", "MS_MorganStanley_ret_1d", "IWM_SmallCap_vol_20d", "XLY_Disc_vol_20d", "CLX_Clorox_vol_20d", "EWJ_Japan_vol_20d", "CMCSA_ret_1d", "NVDA_vol_20d", "EWA_Australia_zscore_60d", "SLB_Schlumberger_ret_5d", "heston_var_ev_h3", "EQIX_Equinix_ret_5d", "T_ret_1d", "TM_Telephone_vol_20d", "NEE_NextEra_ret_20d", "Brent_Oil_FRED_ret_20d", "HD_ret_1d", "VOD_Vodafone_zscore_60d", "TM_Telephone_ret_1d", "3M_ret_5d", "US7Y_Rate_ret_20d"], "is_new": true}, {"model_id": "new_h10_NORMAL_XGBoost_N25_t4", "algo": "XGBoost", "regime": "NORMAL", "horizon": 10, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "LMT_LockheedMartin_vol_20d", "SCHW_Schwab_ret_5d", "HangSeng_HK_vol_20d", "US3Y_Rate_ret_5d", "VRP_ma5", "US1Y_Rate_ret_5d", "XOM_ret_1d", "GD_GeneralDynamics_zscore_60d", "PFE_ret_1d", "EWQ_France_zscore_60d", "T_ret_1d", "SJM_JM_Smucker_ret_5d", "BLK_BlackRock_zscore_60d", "EQR_Equity_ret_1d", "LOW_Lowes_ret_20d", "HangSeng_HK_ret_5d", "EXC_Exelon_zscore_60d", "GE_ret_1d", "AMGN_Amgen_ret_1d", "TM_Telephone_vol_20d", "NOC_Northrop_ret_20d", "CMCSA_ret_1d", "SBUX_ret_5d"], "is_new": true}, {"model_id": "new_h10_NORMAL_XGBoost_N25_t5", "algo": "XGBoost", "regime": "NORMAL", "horizon": 10, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "MSTR_Bitcoin3_ret_1d", "SBUX_ret_5d", "QQQ_vol_20d", "XLF_Fin_vol_20d", "EWY_Korea_zscore_60d", "DHR_ret_1d", "AORD_AUS_zscore_60d", "GD_GeneralDynamics_zscore_60d", "US3M_Rate_zscore_60d", "AMZN_ret_5d", "US7Y_Rate_ret_20d", "AMD_ret_1d", "XLY_Disc_vol_20d", "MO_AltriaMG_ret_1d", "US5Y_Rate_ret_5d", "HD_zscore_60d", "XLK_Tech_zscore_60d", "spx_abs_ret_max_5d", "HangSeng_HK_vol_20d", "BLK_BlackRock_zscore_60d", "LOW_Lowes_ret_5d", "Nikkei_Japan_zscore_60d", "MRK_Merck_zscore_60d"], "is_new": true}, {"model_id": "new_h10_NORMAL_XGBoost_N25_t6", "algo": "XGBoost", "regime": "NORMAL", "horizon": 10, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "CTAS_Cintas_vol_20d", "SBUX_zscore_60d", "CPB_CampbellSoup_ret_20d", "TM_Telephone_ret_1d", "DHR_ret_1d", "EWM_Malaysia_vol_20d", "DAX_Germany_vol_20d", "EWA_Australia_zscore_60d", "AMT_AmericanTower_ret_1d", "AVB_AvalonBay_zscore_60d", "vix_acceleration_1d", "TGT_Target_zscore_60d", "heston_var_ev_h7", "LUV_SouthwestAir_ret_5d", "spx_abs_ret_max_5d", "SBUX_vol_20d", "PAYX_Paychex_ret_20d", "TXN_vol_20d", "XLK_Tech_zscore_60d", "AXP_Amex_vol_20d", "EOG_EOGResources_ret_5d", "DAX_Germany_zscore_60d", "MSTR_Bitcoin3_ret_20d"], "is_new": true}, {"model_id": "new_h10_NORMAL_XGBoost_N25_t7", "algo": "XGBoost", "regime": "NORMAL", "horizon": 10, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "DE_Deere_ret_5d", "SBUX_vol_20d", "PPL_PPL_ret_1d", "vix_mean_abs_ret_5d", "EWM_Malaysia_vol_20d", "GD_GeneralDynamics_zscore_60d", "SLB_Schlumberger_ret_5d", "XLK_Tech_zscore_60d", "BLK_BlackRock_zscore_60d", "DHR_vol_20d", "MSTR_Bitcoin3_ret_20d", "EFFR_ret_1d", "CPB_CampbellSoup_zscore_60d", "US30Y_Rate_ret_20d", "Brent_Oil_FRED_ret_20d", "WTI_Oil_FRED_zscore_60d", "US1Y_Rate_ret_5d", "VRP_ma5", "EFFR_vol_20d", "HangSeng_HK_ret_1d", "AORD_AUS_zscore_60d", "QQQ_vol_20d", "SCHW_Schwab_ret_5d"], "is_new": true}, {"model_id": "new_h10_NORMAL_XGBoost_N30_t0", "algo": "XGBoost", "regime": "NORMAL", "horizon": 10, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "US1Y_Rate_ret_20d", "MS_MorganStanley_ret_1d", "heston_var_ev_h5", "EWQ_France_ret_20d", "NVDA_vol_20d", "IBEX_Spain_ret_20d", "INTC_ret_1d", "TGT_Target_zscore_60d", "XOM_ret_1d", "heston_var_ev_h7", "GE_ret_1d", "heston_var_ev_h3", "Nikkei_Japan_vol_20d", "QQQ_vol_20d", "CTAS_Cintas_vol_20d", "Michigan_Sentiment_ret_20d", "AMGN_Amgen_ret_1d", "TED_Spread_zscore_60d", "AMT_AmericanTower_ret_1d", "XLK_Tech_zscore_60d", "GILD_Gilead_ret_20d", "XLY_Disc_vol_20d", "DOW_Price_zscore_60d", "HangSeng_HK_ret_5d", "DHR_vol_20d", "DAX_Germany_zscore_60d", "CCI_CrownCastle_vol_20d", "EWL_Switzerland_zscore_60d"], "is_new": true}, {"model_id": "new_h10_NORMAL_XGBoost_N30_t1", "algo": "XGBoost", "regime": "NORMAL", "horizon": 10, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "US3M_Rate_vol_20d", "AXP_Amex_vol_20d", "EMR_Emerson_ret_20d", "EWA_Australia_ret_1d", "3M_ret_5d", "PAYX_Paychex_zscore_60d", "LMT_LockheedMartin_vol_20d", "spx_abs_ret_max_5d", "EXC_Exelon_zscore_60d", "XLB_Materials_zscore_60d", "CTAS_Cintas_vol_20d", "NFCI_ret_5d", "T_ret_1d", "HD_zscore_60d", "DIS_vol_20d", "Nikkei_Japan_vol_20d", "BA_ret_1d", "EWY_Korea_zscore_60d", "heston_var_ev_h5", "TGT_Target_zscore_60d", "PG_ret_20d", "LOW_Lowes_ret_20d", "EWQ_France_ret_20d", "US7Y_Rate_ret_20d", "XOM_ret_20d", "LLY_zscore_60d", "US3M_Rate_zscore_60d", "US1Y_Rate_ret_5d"], "is_new": true}, {"model_id": "new_h10_NORMAL_XGBoost_N30_t2", "algo": "XGBoost", "regime": "NORMAL", "horizon": 10, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EMR_Emerson_ret_20d", "ENB_EnbridgeInc_ret_1d", "HangSeng_HK_ret_1d", "US3M_Rate_vol_20d", "BA_ret_1d", "BTI_BritishAmerican_ret_5d", "PCAR_PaccarInc_ret_5d", "US6M_Rate_ret_20d", "EWH_HongKong_ret_5d", "MSTR_Bitcoin3_ret_1d", "EWL_Switzerland_zscore_60d", "Industrial_Production_zscore_60d", "spx_vol_5d", "PLD_Prologis_ret_5d", "AVB_AvalonBay_zscore_60d", "HUM_Humana_ret_5d", "BLK_BlackRock_zscore_60d", "T10Y2Y_Spread_ret_5d", "spx_abs_ret_max_5d", "NEE_NextEra_ret_20d", "EWQ_France_ret_20d", "VOD_Vodafone_zscore_60d", "DHR_ret_1d", "SLB_Schlumberger_ret_5d", "vix_acceleration_1d", "EWQ_France_zscore_60d", "TGT_Target_zscore_60d", "ES_Evergy_ret_1d"], "is_new": true}, {"model_id": "new_h10_NORMAL_XGBoost_N30_t3", "algo": "XGBoost", "regime": "NORMAL", "horizon": 10, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EFFR_vol_20d", "CCI_CrownCastle_vol_20d", "ES_Evergy_ret_1d", "XLY_Disc_vol_20d", "US7Y_Rate_ret_20d", "gjr_condvar_h1", "US3M_Rate_vol_20d", "HangSeng_HK_vol_20d", "FedFunds_zscore_60d", "IWM_SmallCap_vol_20d", "BA_ret_1d", "QQQ_vol_20d", "US1Y_Rate_ret_5d", "heston_var_ev_h3", "VVIX_ret_20d", "3M_ret_5d", "DE_Deere_vol_20d", "Michigan_Sentiment_ret_20d", "LOW_Lowes_ret_5d", "vix_acceleration_1d", "SBUX_ret_5d", "NVDA_vol_20d", "XLV_Health_zscore_60d", "DAX_Germany_zscore_60d", "CPB_CampbellSoup_ret_20d", "HangSeng_HK_ret_1d", "EWJ_Japan_vol_20d", "BLK_BlackRock_zscore_60d"], "is_new": true}, {"model_id": "new_h10_NORMAL_XGBoost_N30_t4", "algo": "XGBoost", "regime": "NORMAL", "horizon": 10, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "MSTR_Bitcoin3_ret_5d", "XOM_ret_20d", "vix_mean_abs_ret_5d", "heston_ev_h3", "SPY_zscore_60d", "NFCI_ret_5d", "ENB_EnbridgeInc_ret_1d", "EWQ_France_ret_20d", "SLB_Schlumberger_ret_1d", "CPB_CampbellSoup_zscore_60d", "DE_Deere_ret_5d", "PAYX_Paychex_vol_20d", "PG_ret_20d", "LLY_zscore_60d", "TM_Telephone_ret_1d", "IBEX_Spain_ret_20d", "EWL_Switzerland_vol_20d", "AMZN_ret_5d", "T10Y2Y_Spread_ret_5d", "WTI_Oil_FRED_zscore_60d", "EOG_EOGResources_vol_20d", "GE_ret_1d", "CTAS_Cintas_vol_20d", "GILD_Gilead_ret_20d", "CPB_CampbellSoup_vol_20d", "EWM_Malaysia_vol_20d", "Nikkei_Japan_vol_20d", "heston_var_ev_h3"], "is_new": true}, {"model_id": "new_h10_NORMAL_XGBoost_N30_t5", "algo": "XGBoost", "regime": "NORMAL", "horizon": 10, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "HangSeng_HK_vol_20d", "Nikkei_Japan_vol_20d", "HD_zscore_60d", "EWA_Australia_ret_1d", "SCHW_Schwab_ret_5d", "EWS_Singapore_ret_5d", "TED_Spread_zscore_60d", "CLX_Clorox_vol_20d", "Michigan_Sentiment_ret_20d", "SBUX_zscore_60d", "HD_ret_20d", "spx_abs_ret_max_5d", "vix_mean_abs_ret_5d", "EMR_Emerson_ret_20d", "SJM_JM_Smucker_ret_1d", "MSTR_Bitcoin3_ret_20d", "TM_Telephone_vol_20d", "MO_AltriaMG_ret_1d", "IYM_BasicMaterials_ret_20d", "CI_Cigna_vol_20d", "HangSeng_HK_ret_1d", "XLK_Tech_zscore_60d", "CPB_CampbellSoup_ret_20d", "EFFR_ret_1d", "spx_vol_5d", "HUM_Humana_ret_5d", "MS_MorganStanley_zscore_60d", "LMT_LockheedMartin_vol_20d"], "is_new": true}, {"model_id": "new_h10_NORMAL_XGBoost_N30_t6", "algo": "XGBoost", "regime": "NORMAL", "horizon": 10, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "CTAS_Cintas_vol_20d", "MS_MorganStanley_ret_5d", "XOM_ret_20d", "vix_mean_abs_ret_5d", "EWA_Australia_ret_1d", "IYR_US_REIT2_zscore_60d", "EFFR_vol_20d", "MSTR_Bitcoin3_ret_5d", "HangSeng_HK_vol_20d", "DOW_Price_zscore_60d", "LMT_LockheedMartin_vol_20d", "XLK_Tech_zscore_60d", "CPB_CampbellSoup_vol_20d", "hmm_p_stress", "HD_zscore_60d", "EWG_Germany_ret_20d", "EWC_Canada_zscore_60d", "US6M_Rate_ret_20d", "QQQ_vol_20d", "EXC_Exelon_zscore_60d", "TM_Telephone_ret_1d", "AORD_AUS_zscore_60d", "VRP_ma5", "SJM_JM_Smucker_ret_1d", "ORCL_zscore_60d", "3M_vol_20d", "ASX_Australia_vol_20d", "Michigan_Sentiment_ret_20d"], "is_new": true}, {"model_id": "new_h10_NORMAL_XGBoost_N30_t7", "algo": "XGBoost", "regime": "NORMAL", "horizon": 10, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "HUM_Humana_ret_5d", "Industrial_Production_zscore_60d", "CLX_Clorox_vol_20d", "SBUX_zscore_60d", "US3Y_Rate_ret_5d", "EQIX_Equinix_ret_5d", "US3M_Rate_zscore_60d", "NFCI_ret_5d", "TGT_Target_zscore_60d", "SBUX_vol_20d", "PAYX_Paychex_ret_20d", "DE_Deere_vol_20d", "US3M_Rate_vol_20d", "BTI_BritishAmerican_ret_20d", "T_ret_1d", "TED_Spread_vol_20d", "vix_mean_abs_ret_5d", "EXC_Exelon_zscore_60d", "EWA_Australia_zscore_60d", "gjr_condvar_h1", "MSTR_Bitcoin3_ret_5d", "MSTR_Bitcoin3_ret_20d", "DOW_Price_zscore_60d", "PG_ret_20d", "PLD_Prologis_ret_5d", "DIS_vol_20d", "SLB_Schlumberger_ret_5d", "MS_MorganStanley_ret_1d"], "is_new": true}, {"model_id": "new_h10_NORMAL_LightGBM_N5_t0", "algo": "LightGBM", "regime": "NORMAL", "horizon": 10, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EMR_Emerson_ret_20d", "EWQ_France_ret_20d", "HangSeng_HK_vol_20d"], "is_new": true}, {"model_id": "new_h10_NORMAL_LightGBM_N5_t1", "algo": "LightGBM", "regime": "NORMAL", "horizon": 10, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWL_Switzerland_vol_20d", "DHR_vol_20d", "EFFR_vol_20d"], "is_new": true}, {"model_id": "new_h10_NORMAL_LightGBM_N5_t2", "algo": "LightGBM", "regime": "NORMAL", "horizon": 10, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "DHR_vol_20d", "HD_zscore_60d", "JNJ_ret_1d"], "is_new": true}, {"model_id": "new_h10_NORMAL_LightGBM_N5_t3", "algo": "LightGBM", "regime": "NORMAL", "horizon": 10, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EMR_Emerson_ret_20d", "TED_Spread_zscore_60d", "HangSeng_HK_vol_20d"], "is_new": true}, {"model_id": "new_h10_NORMAL_LightGBM_N5_t4", "algo": "LightGBM", "regime": "NORMAL", "horizon": 10, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "SBUX_vol_20d", "HangSeng_HK_ret_1d", "XLF_Fin_vol_20d"], "is_new": true}, {"model_id": "new_h10_NORMAL_LightGBM_N5_t5", "algo": "LightGBM", "regime": "NORMAL", "horizon": 10, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "US3Y_Rate_ret_5d", "EWH_HongKong_ret_5d", "BLK_BlackRock_zscore_60d"], "is_new": true}, {"model_id": "new_h10_NORMAL_LightGBM_N5_t6", "algo": "LightGBM", "regime": "NORMAL", "horizon": 10, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "Core_PCE_zscore_60d", "EWJ_Japan_vol_20d", "T10Y2Y_Spread_ret_5d"], "is_new": true}, {"model_id": "new_h10_NORMAL_LightGBM_N5_t7", "algo": "LightGBM", "regime": "NORMAL", "horizon": 10, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "HD_ret_1d", "WTI_Oil_FRED_zscore_60d", "gjr_condvar_h1"], "is_new": true}, {"model_id": "new_h10_NORMAL_LightGBM_N8_t0", "algo": "LightGBM", "regime": "NORMAL", "horizon": 10, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "IWM_SmallCap_vol_20d", "INTC_ret_5d", "ASX_Australia_vol_20d", "SLB_Schlumberger_ret_5d", "CTAS_Cintas_vol_20d", "MO_AltriaMG_ret_1d"], "is_new": true}, {"model_id": "new_h10_NORMAL_LightGBM_N8_t1", "algo": "LightGBM", "regime": "NORMAL", "horizon": 10, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "ASX_Australia_ret_5d", "CPB_CampbellSoup_vol_20d", "MS_MorganStanley_ret_5d", "PAYX_Paychex_ret_20d", "EOG_EOGResources_ret_5d", "IYM_BasicMaterials_ret_20d"], "is_new": true}, {"model_id": "new_h10_NORMAL_LightGBM_N8_t2", "algo": "LightGBM", "regime": "NORMAL", "horizon": 10, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "Brent_Oil_FRED_ret_20d", "T10Y2Y_Spread_ret_5d", "EWA_Australia_ret_1d", "ORCL_vol_20d", "ITT_ITTInc_ret_5d", "CPB_CampbellSoup_ret_20d"], "is_new": true}, {"model_id": "new_h10_NORMAL_LightGBM_N8_t3", "algo": "LightGBM", "regime": "NORMAL", "horizon": 10, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWH_HongKong_ret_5d", "TED_Spread_zscore_60d", "SJM_JM_Smucker_ret_1d", "AXP_Amex_vol_20d", "SBUX_ret_5d", "vix_mean_abs_ret_5d"], "is_new": true}, {"model_id": "new_h10_NORMAL_LightGBM_N8_t4", "algo": "LightGBM", "regime": "NORMAL", "horizon": 10, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "US3M_Rate_zscore_60d", "EFFR_vol_20d", "EMR_Emerson_ret_20d", "GD_GeneralDynamics_zscore_60d", "US3Y_Rate_ret_5d", "IBEX_Spain_ret_20d"], "is_new": true}, {"model_id": "new_h10_NORMAL_LightGBM_N8_t5", "algo": "LightGBM", "regime": "NORMAL", "horizon": 10, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "3M_ret_5d", "CTAS_Cintas_vol_20d", "hmm_p_stress", "SPY_zscore_60d", "TXN_vol_20d", "JNJ_ret_1d"], "is_new": true}, {"model_id": "new_h10_NORMAL_LightGBM_N8_t6", "algo": "LightGBM", "regime": "NORMAL", "horizon": 10, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "GD_GeneralDynamics_zscore_60d", "LUV_SouthwestAir_ret_5d", "EOG_EOGResources_ret_5d", "LOW_Lowes_ret_5d", "EWG_Germany_vol_20d", "M_Macys_vol_20d"], "is_new": true}, {"model_id": "new_h10_NORMAL_LightGBM_N8_t7", "algo": "LightGBM", "regime": "NORMAL", "horizon": 10, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "PCAR_PaccarInc_ret_5d", "LUV_SouthwestAir_ret_5d", "CI_Cigna_vol_20d", "EXC_Exelon_zscore_60d", "CPB_CampbellSoup_ret_20d", "AMD_ret_1d"], "is_new": true}, {"model_id": "new_h10_NORMAL_LightGBM_N10_t0", "algo": "LightGBM", "regime": "NORMAL", "horizon": 10, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "GD_GeneralDynamics_zscore_60d", "EOG_EOGResources_vol_20d", "US7Y_Rate_ret_20d", "AVB_AvalonBay_zscore_60d", "PLD_Prologis_ret_5d", "EWQ_France_ret_20d", "EWM_Malaysia_ret_1d", "SPY_zscore_60d"], "is_new": true}, {"model_id": "new_h10_NORMAL_LightGBM_N10_t1", "algo": "LightGBM", "regime": "NORMAL", "horizon": 10, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "heston_var_ev_h7", "MSTR_Bitcoin3_ret_20d", "EOG_EOGResources_vol_20d", "EWA_Australia_zscore_60d", "Core_PCE_zscore_60d", "US7Y_Rate_ret_20d", "EWG_Germany_ret_20d", "XLV_Health_zscore_60d"], "is_new": true}, {"model_id": "new_h10_NORMAL_LightGBM_N10_t2", "algo": "LightGBM", "regime": "NORMAL", "horizon": 10, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "BA_ret_1d", "heston_ev_h3", "FedFunds_zscore_60d", "CI_Cigna_vol_20d", "DE_Deere_vol_20d", "CMCSA_ret_1d", "Industrial_Production_zscore_60d", "AMT_AmericanTower_ret_1d"], "is_new": true}, {"model_id": "new_h10_NORMAL_LightGBM_N10_t3", "algo": "LightGBM", "regime": "NORMAL", "horizon": 10, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "IBEX_Spain_ret_20d", "US6M_Rate_ret_20d", "MRK_Merck_zscore_60d", "Core_PCE_zscore_60d", "ES_Evergy_ret_1d", "VRP_ma5", "FedFunds_zscore_60d", "heston_var_ev_h3"], "is_new": true}, {"model_id": "new_h10_NORMAL_LightGBM_N10_t4", "algo": "LightGBM", "regime": "NORMAL", "horizon": 10, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "US5Y_Rate_ret_5d", "hmm_p_stress", "CTAS_Cintas_vol_20d", "US6M_Rate_ret_20d", "SJM_JM_Smucker_ret_5d", "Michigan_Sentiment_ret_20d", "EOG_EOGResources_ret_5d", "NFCI_ret_5d"], "is_new": true}, {"model_id": "new_h10_NORMAL_LightGBM_N10_t5", "algo": "LightGBM", "regime": "NORMAL", "horizon": 10, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "SO_SouthernCo_ret_5d", "EOG_EOGResources_ret_5d", "NEE_NextEra_ret_20d", "IYM_BasicMaterials_ret_20d", "INTC_ret_1d", "AXP_Amex_ret_20d", "spx_momentum_3d", "US3M_Rate_vol_20d"], "is_new": true}, {"model_id": "new_h10_NORMAL_LightGBM_N10_t6", "algo": "LightGBM", "regime": "NORMAL", "horizon": 10, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "Core_CPI_zscore_60d", "SLB_Schlumberger_ret_1d", "IBEX_Spain_ret_20d", "Michigan_Sentiment_ret_20d", "XOM_ret_20d", "SCHW_Schwab_ret_5d", "BDX_Becton_Dickinson_ret_20d", "spx_momentum_3d"], "is_new": true}, {"model_id": "new_h10_NORMAL_LightGBM_N10_t7", "algo": "LightGBM", "regime": "NORMAL", "horizon": 10, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "IBEX_Spain_ret_20d", "spx_abs_ret_max_5d", "HD_zscore_60d", "BDX_Becton_Dickinson_ret_20d", "Brent_Oil_FRED_ret_5d", "EOG_EOGResources_vol_20d", "vix_mean_abs_ret_5d", "BTI_BritishAmerican_ret_5d"], "is_new": true}, {"model_id": "new_h10_NORMAL_LightGBM_N12_t0", "algo": "LightGBM", "regime": "NORMAL", "horizon": 10, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "AMGN_Amgen_ret_1d", "Nikkei_Japan_zscore_60d", "JNJ_ret_1d", "TM_Telephone_ret_1d", "VOD_Vodafone_zscore_60d", "TED_Spread_vol_20d", "SLB_Schlumberger_ret_5d", "HD_ret_1d", "3M_vol_20d", "MSTR_Bitcoin3_ret_5d"], "is_new": true}, {"model_id": "new_h10_NORMAL_LightGBM_N12_t1", "algo": "LightGBM", "regime": "NORMAL", "horizon": 10, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "Brent_Oil_FRED_ret_5d", "TGT_Target_zscore_60d", "CCI_CrownCastle_vol_20d", "EWM_Malaysia_ret_1d", "EXC_Exelon_ret_1d", "BTI_BritishAmerican_ret_20d", "AMD_ret_1d", "ITT_ITTInc_ret_5d", "WTI_Oil_FRED_zscore_60d", "SBUX_zscore_60d"], "is_new": true}, {"model_id": "new_h10_NORMAL_LightGBM_N12_t2", "algo": "LightGBM", "regime": "NORMAL", "horizon": 10, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "heston_ev_h3", "NFCI_ret_5d", "ES_Evergy_ret_1d", "vix_mean_abs_ret_5d", "HUM_Humana_ret_5d", "LOW_Lowes_ret_5d", "DE_Deere_ret_5d", "PAYX_Paychex_zscore_60d", "ORCL_vol_20d", "BTI_BritishAmerican_ret_5d"], "is_new": true}, {"model_id": "new_h10_NORMAL_LightGBM_N12_t3", "algo": "LightGBM", "regime": "NORMAL", "horizon": 10, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "SPY_zscore_60d", "SCHW_Schwab_ret_5d", "DE_Deere_vol_20d", "T10Y2Y_Spread_ret_5d", "BDX_Becton_Dickinson_ret_20d", "3M_ret_5d", "US30Y_Rate_ret_20d", "TED_Spread_vol_20d", "EWH_HongKong_ret_5d", "VRP_ma5"], "is_new": true}, {"model_id": "new_h10_NORMAL_LightGBM_N12_t4", "algo": "LightGBM", "regime": "NORMAL", "horizon": 10, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "Retail_Sales_zscore_60d", "ES_Evergy_ret_1d", "WTI_Oil_FRED_zscore_60d", "US5Y_Rate_ret_5d", "XLY_Disc_vol_20d", "DOW_Price_zscore_60d", "EWC_Canada_zscore_60d", "EWG_Germany_ret_20d", "IYM_BasicMaterials_ret_20d", "MSTR_Bitcoin3_ret_5d"], "is_new": true}, {"model_id": "new_h10_NORMAL_LightGBM_N12_t5", "algo": "LightGBM", "regime": "NORMAL", "horizon": 10, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "ASX_Australia_vol_20d", "EWQ_France_zscore_60d", "EMR_Emerson_ret_20d", "EQR_Equity_ret_1d", "IWM_SmallCap_vol_20d", "NEE_NextEra_ret_20d", "HUM_Humana_ret_5d", "SPY_zscore_60d", "EWG_Germany_vol_20d", "M_Macys_vol_20d"], "is_new": true}, {"model_id": "new_h10_NORMAL_LightGBM_N12_t6", "algo": "LightGBM", "regime": "NORMAL", "horizon": 10, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "Brent_Oil_FRED_ret_20d", "Nikkei_Japan_vol_20d", "HD_ret_5d", "ASX_Australia_vol_20d", "MS_MorganStanley_ret_5d", "spx_abs_ret_max_5d", "DAX_Germany_zscore_60d", "US5Y_Rate_ret_5d", "EWY_Korea_zscore_60d", "LMT_LockheedMartin_ret_1d"], "is_new": true}, {"model_id": "new_h10_NORMAL_LightGBM_N12_t7", "algo": "LightGBM", "regime": "NORMAL", "horizon": 10, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "CPB_CampbellSoup_ret_5d", "PLD_Prologis_ret_5d", "SBUX_ret_5d", "heston_var_ev_h7", "IYM_BasicMaterials_ret_20d", "SBUX_zscore_60d", "IYR_US_REIT2_zscore_60d", "DHR_vol_20d", "LMT_LockheedMartin_vol_20d", "CTAS_Cintas_vol_20d"], "is_new": true}, {"model_id": "new_h10_NORMAL_LightGBM_N15_t0", "algo": "LightGBM", "regime": "NORMAL", "horizon": 10, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "BTI_BritishAmerican_ret_20d", "TED_Spread_vol_20d", "DOW_Price_zscore_60d", "EFFR_ret_1d", "ENB_EnbridgeInc_ret_1d", "TM_Telephone_vol_20d", "CPB_CampbellSoup_ret_5d", "AMD_ret_1d", "SBUX_vol_20d", "EWL_Switzerland_zscore_60d", "EWA_Australia_ret_1d", "Nikkei_Japan_vol_20d", "SJM_JM_Smucker_ret_1d"], "is_new": true}, {"model_id": "new_h10_NORMAL_LightGBM_N15_t1", "algo": "LightGBM", "regime": "NORMAL", "horizon": 10, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "DE_Deere_vol_20d", "EQIX_Equinix_ret_5d", "EOG_EOGResources_vol_20d", "HangSeng_HK_ret_5d", "MSTR_Bitcoin3_ret_20d", "EFFR_vol_20d", "QQQ_vol_20d", "Core_PCE_zscore_60d", "TXN_vol_20d", "CPB_CampbellSoup_vol_20d", "3M_ret_5d", "BDX_Becton_Dickinson_ret_20d", "heston_var_ev_h3"], "is_new": true}, {"model_id": "new_h10_NORMAL_LightGBM_N15_t2", "algo": "LightGBM", "regime": "NORMAL", "horizon": 10, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "PFE_ret_1d", "AMZN_ret_5d", "SBUX_ret_5d", "XLV_Health_zscore_60d", "MRK_Merck_zscore_60d", "heston_var_ev_h3", "EWM_Malaysia_vol_20d", "Nikkei_Japan_vol_20d", "US30Y_Rate_ret_20d", "PAYX_Paychex_ret_20d", "AMT_AmericanTower_ret_1d", "DE_Deere_ret_5d", "TED_Spread_vol_20d"], "is_new": true}, {"model_id": "new_h10_NORMAL_LightGBM_N15_t3", "algo": "LightGBM", "regime": "NORMAL", "horizon": 10, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "XLB_Materials_zscore_60d", "SBUX_ret_5d", "heston_var_ev_h3", "Michigan_Sentiment_ret_20d", "PPL_PPL_ret_1d", "LOW_Lowes_ret_5d", "AXP_Amex_vol_20d", "IYM_BasicMaterials_ret_20d", "Nikkei_Japan_vol_20d", "SCHW_Schwab_ret_5d", "ITT_ITTInc_ret_5d", "EWM_Malaysia_zscore_60d", "XLF_Fin_vol_20d"], "is_new": true}, {"model_id": "new_h10_NORMAL_LightGBM_N15_t4", "algo": "LightGBM", "regime": "NORMAL", "horizon": 10, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "Industrial_Production_zscore_60d", "3M_ret_5d", "EWA_Australia_zscore_60d", "MSTR_Bitcoin3_ret_5d", "EWJ_Japan_vol_20d", "JNJ_ret_1d", "EWC_Canada_zscore_60d", "EQR_Equity_ret_1d", "HUM_Humana_ret_5d", "EWG_Germany_ret_20d", "US30Y_Rate_ret_20d", "EXC_Exelon_zscore_60d", "INTC_ret_5d"], "is_new": true}, {"model_id": "new_h10_NORMAL_LightGBM_N15_t5", "algo": "LightGBM", "regime": "NORMAL", "horizon": 10, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWA_Australia_ret_1d", "PPL_PPL_ret_1d", "hmm_p_stress", "BDX_Becton_Dickinson_ret_20d", "heston_var_ev_h7", "EWS_Singapore_ret_5d", "US5Y_Rate_ret_5d", "LMT_LockheedMartin_ret_1d", "EWY_Korea_zscore_60d", "SJM_JM_Smucker_ret_5d", "EWC_Canada_zscore_60d", "EQIX_Equinix_ret_5d", "heston_ev_h3"], "is_new": true}, {"model_id": "new_h10_NORMAL_LightGBM_N15_t6", "algo": "LightGBM", "regime": "NORMAL", "horizon": 10, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "PAYX_Paychex_vol_20d", "US3Y_Rate_ret_5d", "AMD_ret_1d", "TGT_Target_zscore_60d", "EMR_Emerson_ret_20d", "CPB_CampbellSoup_ret_5d", "Brent_Oil_FRED_ret_20d", "EQR_Equity_ret_1d", "ORCL_zscore_60d", "DOW_Price_zscore_60d", "VRP_ma5", "EWG_Germany_vol_20d", "EWY_Korea_zscore_60d"], "is_new": true}, {"model_id": "new_h10_NORMAL_LightGBM_N15_t7", "algo": "LightGBM", "regime": "NORMAL", "horizon": 10, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "PAYX_Paychex_vol_20d", "NEE_NextEra_ret_20d", "AXP_Amex_ret_20d", "ENB_EnbridgeInc_ret_1d", "US1Y_Rate_ret_20d", "3M_ret_5d", "HangSeng_HK_ret_5d", "EWC_Canada_zscore_60d", "AMT_AmericanTower_ret_1d", "MS_MorganStanley_ret_5d", "Michigan_Sentiment_ret_20d", "EWQ_France_ret_20d", "SLB_Schlumberger_ret_5d"], "is_new": true}, {"model_id": "new_h10_NORMAL_LightGBM_N20_t0", "algo": "LightGBM", "regime": "NORMAL", "horizon": 10, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "QQQ_vol_20d", "AMT_AmericanTower_ret_1d", "EWG_Germany_vol_20d", "CMCSA_ret_1d", "EXC_Exelon_ret_1d", "LOW_Lowes_ret_5d", "Brent_Oil_FRED_ret_5d", "SJM_JM_Smucker_ret_5d", "heston_var_ev_h3", "US7Y_Rate_ret_20d", "CLX_Clorox_vol_20d", "Nikkei_Japan_vol_20d", "TM_Telephone_vol_20d", "NVDA_vol_20d", "Retail_Sales_zscore_60d", "XLK_Tech_zscore_60d", "MSTR_Bitcoin3_ret_20d", "SBUX_zscore_60d"], "is_new": true}, {"model_id": "new_h10_NORMAL_LightGBM_N20_t1", "algo": "LightGBM", "regime": "NORMAL", "horizon": 10, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "M_Macys_vol_20d", "ENB_EnbridgeInc_ret_1d", "XOM_ret_1d", "DIS_vol_20d", "TED_Spread_vol_20d", "VOD_Vodafone_zscore_60d", "DE_Deere_vol_20d", "EWQ_France_zscore_60d", "spx_vol_5d", "SBUX_vol_20d", "BTI_BritishAmerican_ret_20d", "hmm_p_stress", "AVB_AvalonBay_zscore_60d", "MS_MorganStanley_ret_5d", "3M_ret_5d", "EWL_Switzerland_zscore_60d", "XLB_Materials_zscore_60d", "heston_var_ev_h5"], "is_new": true}, {"model_id": "new_h10_NORMAL_LightGBM_N20_t2", "algo": "LightGBM", "regime": "NORMAL", "horizon": 10, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "US1Y_Rate_ret_20d", "EWL_Switzerland_vol_20d", "US5Y_Rate_ret_5d", "DAX_Germany_vol_20d", "T10Y2Y_Spread_ret_5d", "AORD_AUS_zscore_60d", "SBUX_vol_20d", "EWM_Malaysia_zscore_60d", "DE_Deere_vol_20d", "AXP_Amex_vol_20d", "WTI_Oil_FRED_zscore_60d", "XLB_Materials_zscore_60d", "US30Y_Rate_ret_20d", "heston_var_ev_h7", "spx_momentum_3d", "PG_ret_20d", "CCI_CrownCastle_vol_20d", "EWY_Korea_zscore_60d"], "is_new": true}, {"model_id": "new_h10_NORMAL_LightGBM_N20_t3", "algo": "LightGBM", "regime": "NORMAL", "horizon": 10, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "AMT_AmericanTower_ret_1d", "Nikkei_Japan_vol_20d", "CLX_Clorox_vol_20d", "EMR_Emerson_ret_20d", "GE_ret_1d", "AORD_AUS_zscore_60d", "Industrial_Production_zscore_60d", "Michigan_Sentiment_ret_20d", "SJM_JM_Smucker_ret_5d", "PLD_Prologis_ret_5d", "MSTR_Bitcoin3_ret_1d", "AMZN_ret_5d", "XLV_Health_zscore_60d", "EXC_Exelon_ret_1d", "PPL_PPL_ret_1d", "HangSeng_HK_vol_20d", "EFFR_vol_20d", "EWG_Germany_vol_20d"], "is_new": true}, {"model_id": "new_h10_NORMAL_LightGBM_N20_t4", "algo": "LightGBM", "regime": "NORMAL", "horizon": 10, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EXC_Exelon_ret_1d", "vix_acceleration_1d", "NFCI_ret_5d", "EWL_Switzerland_zscore_60d", "US3M_Rate_zscore_60d", "MSTR_Bitcoin3_ret_1d", "AMD_ret_5d", "XLY_Disc_vol_20d", "QQQ_vol_20d", "US5Y_Rate_ret_5d", "JNJ_ret_1d", "EFFR_ret_1d", "heston_var_ev_h7", "LOW_Lowes_ret_5d", "LOW_Lowes_ret_20d", "EQR_Equity_ret_1d", "Brent_Oil_FRED_ret_5d", "IBEX_Spain_ret_20d"], "is_new": true}, {"model_id": "new_h10_NORMAL_LightGBM_N20_t5", "algo": "LightGBM", "regime": "NORMAL", "horizon": 10, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EFFR_ret_1d", "INTC_ret_1d", "XLF_Fin_vol_20d", "3M_ret_5d", "AVB_AvalonBay_zscore_60d", "Retail_Sales_zscore_60d", "Core_CPI_zscore_60d", "AXP_Amex_ret_20d", "VVIX_ret_20d", "heston_ev_h3", "SJM_JM_Smucker_ret_1d", "EMR_Emerson_ret_20d", "SLB_Schlumberger_ret_5d", "EXC_Exelon_ret_1d", "CCI_CrownCastle_vol_20d", "SCHW_Schwab_ret_5d", "EWM_Malaysia_vol_20d", "MSTR_Bitcoin3_ret_5d"], "is_new": true}, {"model_id": "new_h10_NORMAL_LightGBM_N20_t6", "algo": "LightGBM", "regime": "NORMAL", "horizon": 10, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "DHR_vol_20d", "HangSeng_HK_ret_5d", "EWA_Australia_zscore_60d", "SLB_Schlumberger_ret_5d", "AVB_AvalonBay_zscore_60d", "BA_ret_1d", "HD_zscore_60d", "Michigan_Sentiment_ret_20d", "XOM_ret_20d", "AMD_ret_1d", "Retail_Sales_zscore_60d", "LLY_zscore_60d", "gjr_condvar_h1", "SJM_JM_Smucker_ret_1d", "CPB_CampbellSoup_ret_5d", "VRP_ma5", "MS_MorganStanley_ret_5d", "TM_Telephone_vol_20d"], "is_new": true}, {"model_id": "new_h10_NORMAL_LightGBM_N20_t7", "algo": "LightGBM", "regime": "NORMAL", "horizon": 10, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "PAYX_Paychex_vol_20d", "ASX_Australia_vol_20d", "HD_ret_1d", "EWM_Malaysia_zscore_60d", "XLK_Tech_zscore_60d", "XOM_ret_20d", "CPB_CampbellSoup_ret_5d", "EWJ_Japan_vol_20d", "DE_Deere_vol_20d", "HD_zscore_60d", "Nikkei_Japan_vol_20d", "SJM_JM_Smucker_ret_1d", "Michigan_Sentiment_ret_20d", "IWM_SmallCap_vol_20d", "SLB_Schlumberger_ret_1d", "CCI_CrownCastle_vol_20d", "PLD_Prologis_ret_5d", "US6M_Rate_ret_20d"], "is_new": true}, {"model_id": "new_h10_NORMAL_LightGBM_N25_t0", "algo": "LightGBM", "regime": "NORMAL", "horizon": 10, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "LLY_zscore_60d", "gjr_condvar_h1", "HD_ret_5d", "T10Y2Y_Spread_ret_5d", "PCAR_PaccarInc_ret_5d", "XLK_Tech_zscore_60d", "DOW_Price_zscore_60d", "IYM_BasicMaterials_ret_20d", "EWM_Malaysia_ret_1d", "CMCSA_ret_1d", "IBEX_Spain_ret_20d", "Brent_Oil_FRED_ret_5d", "BTI_BritishAmerican_ret_20d", "DHR_ret_1d", "HD_ret_1d", "PAYX_Paychex_vol_20d", "US30Y_Rate_ret_20d", "US1Y_Rate_ret_5d", "hmm_p_stress", "GD_GeneralDynamics_zscore_60d", "US5Y_Rate_ret_5d", "LOW_Lowes_ret_5d", "TED_Spread_vol_20d"], "is_new": true}, {"model_id": "new_h10_NORMAL_LightGBM_N25_t1", "algo": "LightGBM", "regime": "NORMAL", "horizon": 10, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "Retail_Sales_zscore_60d", "TED_Spread_zscore_60d", "Brent_Oil_FRED_ret_20d", "CPB_CampbellSoup_ret_20d", "HD_ret_5d", "BTI_BritishAmerican_ret_20d", "CPB_CampbellSoup_ret_5d", "CMCSA_ret_1d", "LLY_zscore_60d", "Core_PCE_zscore_60d", "SO_SouthernCo_ret_5d", "GILD_Gilead_ret_20d", "BA_ret_1d", "MSTR_Bitcoin3_ret_20d", "EWH_HongKong_ret_5d", "IWM_SmallCap_vol_20d", "ASX_Australia_ret_5d", "SLB_Schlumberger_ret_1d", "QQQ_vol_20d", "US1Y_Rate_ret_5d", "NVDA_vol_20d", "US30Y_Rate_ret_20d", "ORCL_zscore_60d"], "is_new": true}, {"model_id": "new_h10_NORMAL_LightGBM_N25_t2", "algo": "LightGBM", "regime": "NORMAL", "horizon": 10, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "XOM_ret_20d", "LMT_LockheedMartin_ret_1d", "ITT_ITTInc_ret_5d", "Industrial_Production_zscore_60d", "SBUX_vol_20d", "AMT_AmericanTower_ret_1d", "INTC_ret_1d", "WTI_Oil_FRED_zscore_60d", "EWM_Malaysia_zscore_60d", "EWC_Canada_zscore_60d", "XLB_Materials_zscore_60d", "heston_ev_h3", "AMD_ret_1d", "VOD_Vodafone_zscore_60d", "EWA_Australia_ret_1d", "NOC_Northrop_ret_20d", "PCAR_PaccarInc_ret_5d", "XLK_Tech_zscore_60d", "AMZN_ret_5d", "PAYX_Paychex_vol_20d", "TM_Telephone_vol_20d", "EWQ_France_ret_20d", "Nikkei_Japan_zscore_60d"], "is_new": true}, {"model_id": "new_h10_NORMAL_LightGBM_N25_t3", "algo": "LightGBM", "regime": "NORMAL", "horizon": 10, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWA_Australia_zscore_60d", "Michigan_Sentiment_ret_20d", "US1Y_Rate_ret_20d", "hmm_p_stress", "heston_var_ev_h3", "EWL_Switzerland_vol_20d", "HangSeng_HK_ret_1d", "vix_acceleration_1d", "XLY_Disc_vol_20d", "CPB_CampbellSoup_vol_20d", "NEE_NextEra_ret_20d", "DHR_ret_1d", "PAYX_Paychex_vol_20d", "EWM_Malaysia_ret_1d", "ASX_Australia_ret_5d", "spx_vol_5d", "EWG_Germany_ret_20d", "CLX_Clorox_vol_20d", "EFFR_vol_20d", "AMGN_Amgen_ret_1d", "heston_var_ev_h5", "SBUX_vol_20d", "BTI_BritishAmerican_ret_20d"], "is_new": true}, {"model_id": "new_h10_NORMAL_LightGBM_N25_t4", "algo": "LightGBM", "regime": "NORMAL", "horizon": 10, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWM_Malaysia_vol_20d", "DE_Deere_ret_5d", "NOC_Northrop_ret_20d", "CPB_CampbellSoup_vol_20d", "XLK_Tech_zscore_60d", "GILD_Gilead_ret_20d", "CCI_CrownCastle_vol_20d", "heston_var_ev_h7", "DHR_vol_20d", "EWS_Singapore_ret_5d", "Nikkei_Japan_zscore_60d", "EWG_Germany_ret_20d", "EWY_Korea_zscore_60d", "SO_SouthernCo_ret_5d", "EWH_HongKong_ret_5d", "HD_ret_5d", "IWM_SmallCap_vol_20d", "DOW_Price_zscore_60d", "SBUX_ret_5d", "BA_ret_1d", "NVDA_vol_20d", "EFFR_ret_1d", "FedFunds_zscore_60d"], "is_new": true}, {"model_id": "new_h10_NORMAL_LightGBM_N25_t5", "algo": "LightGBM", "regime": "NORMAL", "horizon": 10, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "LLY_zscore_60d", "EWQ_France_ret_20d", "EWL_Switzerland_vol_20d", "EWS_Singapore_ret_5d", "WTI_Oil_FRED_zscore_60d", "AMD_ret_5d", "EFFR_vol_20d", "VVIX_ret_20d", "M_Macys_vol_20d", "MO_AltriaMG_ret_1d", "CMCSA_ret_1d", "HangSeng_HK_ret_5d", "Nikkei_Japan_zscore_60d", "ES_Evergy_ret_1d", "Core_PCE_zscore_60d", "EWM_Malaysia_ret_1d", "spx_abs_ret_max_5d", "EWM_Malaysia_vol_20d", "MSTR_Bitcoin3_ret_5d", "EMR_Emerson_ret_20d", "CTAS_Cintas_vol_20d", "NOC_Northrop_ret_20d", "US3M_Rate_zscore_60d"], "is_new": true}, {"model_id": "new_h10_NORMAL_LightGBM_N25_t6", "algo": "LightGBM", "regime": "NORMAL", "horizon": 10, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWM_Malaysia_vol_20d", "US3M_Rate_zscore_60d", "AMD_ret_5d", "EWL_Switzerland_zscore_60d", "heston_var_ev_h3", "EWG_Germany_vol_20d", "HD_ret_5d", "SLB_Schlumberger_ret_5d", "EFFR_ret_1d", "NEE_NextEra_ret_20d", "EXC_Exelon_ret_1d", "IYR_US_REIT2_zscore_60d", "EQIX_Equinix_ret_5d", "GILD_Gilead_ret_20d", "spx_vol_5d", "AMZN_ret_5d", "Industrial_Production_zscore_60d", "SBUX_ret_5d", "EWJ_Japan_vol_20d", "LOW_Lowes_ret_20d", "SBUX_zscore_60d", "WTI_Oil_FRED_zscore_60d", "Retail_Sales_zscore_60d"], "is_new": true}, {"model_id": "new_h10_NORMAL_LightGBM_N25_t7", "algo": "LightGBM", "regime": "NORMAL", "horizon": 10, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "AORD_AUS_zscore_60d", "PG_ret_20d", "XOM_ret_1d", "Industrial_Production_zscore_60d", "BTI_BritishAmerican_ret_20d", "ITT_ITTInc_ret_5d", "ENB_EnbridgeInc_ret_1d", "PLD_Prologis_ret_5d", "AXP_Amex_vol_20d", "US3Y_Rate_ret_5d", "SLB_Schlumberger_ret_5d", "EOG_EOGResources_ret_5d", "EWQ_France_zscore_60d", "TED_Spread_zscore_60d", "AMD_ret_5d", "PCAR_PaccarInc_ret_5d", "SBUX_ret_5d", "EWA_Australia_zscore_60d", "LUV_SouthwestAir_ret_5d", "US1Y_Rate_ret_5d", "GD_GeneralDynamics_zscore_60d", "NOC_Northrop_ret_20d", "PPL_PPL_ret_1d"], "is_new": true}, {"model_id": "new_h10_NORMAL_LightGBM_N30_t0", "algo": "LightGBM", "regime": "NORMAL", "horizon": 10, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "ASX_Australia_vol_20d", "CTAS_Cintas_vol_20d", "SPY_zscore_60d", "US30Y_Rate_ret_20d", "NEE_NextEra_ret_20d", "SCHW_Schwab_ret_5d", "US1Y_Rate_ret_5d", "EQIX_Equinix_ret_5d", "heston_ev_h3", "spx_abs_ret_max_5d", "IYM_BasicMaterials_ret_20d", "Brent_Oil_FRED_ret_20d", "MRK_Merck_zscore_60d", "3M_ret_5d", "LOW_Lowes_ret_20d", "SLB_Schlumberger_ret_5d", "VVIX_ret_20d", "EWY_Korea_ret_20d", "LMT_LockheedMartin_ret_1d", "BLK_BlackRock_zscore_60d", "US6M_Rate_ret_20d", "EWL_Switzerland_zscore_60d", "PLD_Prologis_ret_5d", "CPB_CampbellSoup_vol_20d", "CPB_CampbellSoup_zscore_60d", "HD_ret_1d", "PAYX_Paychex_vol_20d", "PG_ret_20d"], "is_new": true}, {"model_id": "new_h10_NORMAL_LightGBM_N30_t1", "algo": "LightGBM", "regime": "NORMAL", "horizon": 10, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EXC_Exelon_zscore_60d", "heston_var_ev_h3", "EOG_EOGResources_vol_20d", "SCHW_Schwab_ret_5d", "PLD_Prologis_ret_5d", "DE_Deere_vol_20d", "ES_Evergy_ret_1d", "CI_Cigna_vol_20d", "EXC_Exelon_ret_1d", "GE_ret_1d", "IBEX_Spain_ret_20d", "EWG_Germany_vol_20d", "TM_Telephone_vol_20d", "PPL_PPL_ret_1d", "HangSeng_HK_vol_20d", "SBUX_zscore_60d", "Industrial_Production_zscore_60d", "CLX_Clorox_vol_20d", "MS_MorganStanley_ret_1d", "EQIX_Equinix_ret_5d", "LMT_LockheedMartin_vol_20d", "XLV_Health_zscore_60d", "PG_ret_20d", "CPB_CampbellSoup_vol_20d", "DIS_vol_20d", "EWH_HongKong_ret_5d", "DE_Deere_ret_5d", "TM_Telephone_ret_1d"], "is_new": true}, {"model_id": "new_h10_NORMAL_LightGBM_N30_t2", "algo": "LightGBM", "regime": "NORMAL", "horizon": 10, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EQIX_Equinix_ret_5d", "XLB_Materials_zscore_60d", "DHR_vol_20d", "GILD_Gilead_ret_20d", "SBUX_ret_5d", "CPB_CampbellSoup_ret_5d", "VRP_ma5", "MS_MorganStanley_zscore_60d", "3M_vol_20d", "HD_ret_5d", "DAX_Germany_vol_20d", "PAYX_Paychex_zscore_60d", "NVDA_vol_20d", "T10Y2Y_Spread_ret_5d", "HD_zscore_60d", "CPB_CampbellSoup_zscore_60d", "IYR_US_REIT2_zscore_60d", "BTI_BritishAmerican_ret_20d", "PAYX_Paychex_vol_20d", "AVB_AvalonBay_zscore_60d", "AORD_AUS_zscore_60d", "AMGN_Amgen_ret_1d", "NOC_Northrop_ret_20d", "MS_MorganStanley_ret_1d", "EFFR_vol_20d", "Nikkei_Japan_vol_20d", "SO_SouthernCo_ret_5d", "JNJ_ret_1d"], "is_new": true}, {"model_id": "new_h10_NORMAL_LightGBM_N30_t3", "algo": "LightGBM", "regime": "NORMAL", "horizon": 10, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "BTI_BritishAmerican_ret_20d", "AMD_ret_1d", "CTAS_Cintas_vol_20d", "Nikkei_Japan_zscore_60d", "SCHW_Schwab_ret_5d", "ENB_EnbridgeInc_ret_1d", "EFFR_vol_20d", "DE_Deere_ret_5d", "CMCSA_ret_1d", "Nikkei_Japan_vol_20d", "spx_momentum_3d", "JNJ_ret_1d", "HD_zscore_60d", "TED_Spread_vol_20d", "PLD_Prologis_ret_5d", "SO_SouthernCo_ret_5d", "Michigan_Sentiment_ret_20d", "GE_ret_1d", "XLF_Fin_vol_20d", "XOM_ret_20d", "TM_Telephone_vol_20d", "CI_Cigna_vol_20d", "LMT_LockheedMartin_vol_20d", "MSTR_Bitcoin3_ret_5d", "MS_MorganStanley_zscore_60d", "US30Y_Rate_ret_20d", "HD_ret_20d", "heston_var_ev_h7"], "is_new": true}, {"model_id": "new_h10_NORMAL_LightGBM_N30_t4", "algo": "LightGBM", "regime": "NORMAL", "horizon": 10, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EOG_EOGResources_ret_5d", "EFFR_vol_20d", "INTC_ret_5d", "HangSeng_HK_ret_1d", "PLD_Prologis_ret_5d", "ASX_Australia_vol_20d", "PAYX_Paychex_zscore_60d", "LUV_SouthwestAir_ret_5d", "US3M_Rate_vol_20d", "SJM_JM_Smucker_ret_5d", "EWM_Malaysia_zscore_60d", "ITT_ITTInc_ret_5d", "DE_Deere_ret_5d", "spx_vol_5d", "spx_abs_ret_max_5d", "TGT_Target_zscore_60d", "VOD_Vodafone_zscore_60d", "AMD_ret_1d", "Michigan_Sentiment_ret_20d", "NVDA_vol_20d", "vix_mean_abs_ret_5d", "IYR_US_REIT2_zscore_60d", "US3Y_Rate_ret_5d", "IYM_BasicMaterials_ret_20d", "SPY_zscore_60d", "DIS_vol_20d", "CLX_Clorox_vol_20d", "EXC_Exelon_ret_1d"], "is_new": true}, {"model_id": "new_h10_NORMAL_LightGBM_N30_t5", "algo": "LightGBM", "regime": "NORMAL", "horizon": 10, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "US3M_Rate_zscore_60d", "EWM_Malaysia_ret_1d", "HangSeng_HK_ret_5d", "CMCSA_ret_1d", "SBUX_vol_20d", "GILD_Gilead_ret_20d", "DHR_vol_20d", "PG_ret_20d", "US3Y_Rate_ret_5d", "SBUX_zscore_60d", "US6M_Rate_ret_20d", "EOG_EOGResources_ret_5d", "LMT_LockheedMartin_vol_20d", "CPB_CampbellSoup_vol_20d", "XLV_Health_zscore_60d", "US1Y_Rate_ret_20d", "NFCI_ret_5d", "EQR_Equity_ret_1d", "heston_var_ev_h5", "US1Y_Rate_ret_5d", "NVDA_vol_20d", "EQIX_Equinix_ret_5d", "spx_momentum_3d", "IYM_BasicMaterials_ret_20d", "TM_Telephone_vol_20d", "BTI_BritishAmerican_ret_5d", "HD_zscore_60d", "SCHW_Schwab_ret_5d"], "is_new": true}, {"model_id": "new_h10_NORMAL_LightGBM_N30_t6", "algo": "LightGBM", "regime": "NORMAL", "horizon": 10, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "SBUX_ret_5d", "EXC_Exelon_ret_1d", "HD_ret_20d", "EFFR_vol_20d", "PAYX_Paychex_zscore_60d", "LMT_LockheedMartin_vol_20d", "Brent_Oil_FRED_ret_20d", "EWG_Germany_vol_20d", "CPB_CampbellSoup_zscore_60d", "ASX_Australia_ret_5d", "CPB_CampbellSoup_ret_20d", "AMD_ret_5d", "CTAS_Cintas_vol_20d", "BDX_Becton_Dickinson_ret_20d", "XOM_ret_20d", "QQQ_vol_20d", "Core_CPI_zscore_60d", "NOC_Northrop_ret_20d", "EWL_Switzerland_zscore_60d", "CCI_CrownCastle_vol_20d", "EWM_Malaysia_vol_20d", "heston_var_ev_h7", "XLF_Fin_vol_20d", "EWC_Canada_zscore_60d", "LMT_LockheedMartin_ret_1d", "XLB_Materials_zscore_60d", "PFE_ret_1d", "US3M_Rate_zscore_60d"], "is_new": true}, {"model_id": "new_h10_NORMAL_LightGBM_N30_t7", "algo": "LightGBM", "regime": "NORMAL", "horizon": 10, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWM_Malaysia_ret_1d", "spx_vol_5d", "DE_Deere_ret_5d", "BDX_Becton_Dickinson_ret_20d", "EXC_Exelon_zscore_60d", "DOW_Price_zscore_60d", "US30Y_Rate_ret_20d", "PAYX_Paychex_ret_20d", "AMD_ret_1d", "EWM_Malaysia_vol_20d", "US3Y_Rate_ret_5d", "MSTR_Bitcoin3_ret_5d", "MSTR_Bitcoin3_ret_1d", "EWY_Korea_ret_20d", "EWM_Malaysia_zscore_60d", "hmm_p_stress", "M_Macys_vol_20d", "NFCI_ret_5d", "DHR_vol_20d", "AMGN_Amgen_ret_1d", "MO_AltriaMG_ret_1d", "DIS_vol_20d", "LUV_SouthwestAir_ret_5d", "MSTR_Bitcoin3_ret_20d", "EFFR_ret_1d", "DAX_Germany_zscore_60d", "MS_MorganStanley_zscore_60d", "IYR_US_REIT2_zscore_60d"], "is_new": true}, {"model_id": "new_h10_NORMAL_GradientBoosting_N5_t0", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 10, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "DHR_vol_20d", "SO_SouthernCo_ret_5d", "BLK_BlackRock_zscore_60d"], "is_new": true}, {"model_id": "new_h10_NORMAL_GradientBoosting_N5_t1", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 10, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "MSTR_Bitcoin3_ret_5d", "US30Y_Rate_ret_20d", "NWL_Newell_ret_20d"], "is_new": true}, {"model_id": "new_h10_NORMAL_GradientBoosting_N5_t2", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 10, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "vix_acceleration_1d", "MRK_Merck_zscore_60d", "BDX_Becton_Dickinson_ret_20d"], "is_new": true}, {"model_id": "new_h10_NORMAL_GradientBoosting_N5_t3", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 10, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "DOW_Price_zscore_60d", "gjr_condvar_h1", "SJM_JM_Smucker_ret_5d"], "is_new": true}, {"model_id": "new_h10_NORMAL_GradientBoosting_N5_t4", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 10, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "GD_GeneralDynamics_zscore_60d", "DOW_Price_zscore_60d", "MRK_Merck_zscore_60d"], "is_new": true}, {"model_id": "new_h10_NORMAL_GradientBoosting_N5_t5", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 10, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "FedFunds_zscore_60d", "NEE_NextEra_ret_20d", "EWH_HongKong_ret_5d"], "is_new": true}, {"model_id": "new_h10_NORMAL_GradientBoosting_N5_t6", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 10, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "IYM_BasicMaterials_ret_20d", "EWS_Singapore_ret_5d", "PFE_ret_1d"], "is_new": true}, {"model_id": "new_h10_NORMAL_GradientBoosting_N5_t7", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 10, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "ES_Evergy_ret_1d", "hmm_p_stress", "JNJ_ret_1d"], "is_new": true}, {"model_id": "new_h10_NORMAL_GradientBoosting_N8_t0", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 10, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "spx_momentum_3d", "3M_ret_5d", "SBUX_vol_20d", "hmm_p_stress", "CLX_Clorox_vol_20d", "CTAS_Cintas_vol_20d"], "is_new": true}, {"model_id": "new_h10_NORMAL_GradientBoosting_N8_t1", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 10, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "Nikkei_Japan_zscore_60d", "MSTR_Bitcoin3_ret_1d", "CPB_CampbellSoup_ret_5d", "US1Y_Rate_ret_20d", "spx_momentum_3d", "hmm_p_stress"], "is_new": true}, {"model_id": "new_h10_NORMAL_GradientBoosting_N8_t2", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 10, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "VOD_Vodafone_zscore_60d", "HUM_Humana_ret_5d", "Retail_Sales_zscore_60d", "DIS_vol_20d", "T_ret_1d", "FedFunds_zscore_60d"], "is_new": true}, {"model_id": "new_h10_NORMAL_GradientBoosting_N8_t3", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 10, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "IYR_US_REIT2_zscore_60d", "EWH_HongKong_ret_5d", "AMT_AmericanTower_ret_1d", "Brent_Oil_FRED_ret_20d", "US3M_Rate_zscore_60d", "T_ret_1d"], "is_new": true}, {"model_id": "new_h10_NORMAL_GradientBoosting_N8_t4", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 10, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "MS_MorganStanley_ret_5d", "AORD_AUS_zscore_60d", "ASX_Australia_ret_5d", "Industrial_Production_zscore_60d", "SO_SouthernCo_ret_5d", "VVIX_ret_20d"], "is_new": true}, {"model_id": "new_h10_NORMAL_GradientBoosting_N8_t5", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 10, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "NEE_NextEra_ret_20d", "AORD_AUS_zscore_60d", "US3Y_Rate_ret_5d", "BTI_BritishAmerican_ret_5d", "CI_Cigna_vol_20d", "PG_ret_20d"], "is_new": true}, {"model_id": "new_h10_NORMAL_GradientBoosting_N8_t6", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 10, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWA_Australia_zscore_60d", "EFFR_vol_20d", "US1Y_Rate_ret_20d", "PAYX_Paychex_zscore_60d", "CPB_CampbellSoup_ret_20d", "LMT_LockheedMartin_ret_1d"], "is_new": true}, {"model_id": "new_h10_NORMAL_GradientBoosting_N8_t7", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 10, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "SBUX_ret_5d", "CTAS_Cintas_vol_20d", "SBUX_vol_20d", "JNJ_ret_1d", "TM_Telephone_ret_1d", "HangSeng_HK_ret_5d"], "is_new": true}, {"model_id": "new_h10_NORMAL_GradientBoosting_N10_t0", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 10, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "US6M_Rate_ret_20d", "SBUX_zscore_60d", "TED_Spread_vol_20d", "T_ret_1d", "CPB_CampbellSoup_vol_20d", "CMCSA_ret_1d", "ASX_Australia_ret_5d", "AMZN_ret_5d"], "is_new": true}, {"model_id": "new_h10_NORMAL_GradientBoosting_N10_t1", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 10, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWY_Korea_ret_20d", "SJM_JM_Smucker_ret_1d", "BLK_BlackRock_zscore_60d", "MSTR_Bitcoin3_ret_20d", "Brent_Oil_FRED_ret_20d", "XLK_Tech_zscore_60d", "PAYX_Paychex_vol_20d", "TM_Telephone_vol_20d"], "is_new": true}, {"model_id": "new_h10_NORMAL_GradientBoosting_N10_t2", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 10, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "PCAR_PaccarInc_ret_5d", "HD_ret_5d", "SBUX_vol_20d", "SBUX_ret_5d", "XLY_Disc_vol_20d", "EWQ_France_ret_20d", "EWL_Switzerland_zscore_60d", "MS_MorganStanley_ret_5d"], "is_new": true}, {"model_id": "new_h10_NORMAL_GradientBoosting_N10_t3", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 10, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "SJM_JM_Smucker_ret_1d", "VOD_Vodafone_zscore_60d", "NVDA_vol_20d", "Core_CPI_zscore_60d", "NEE_NextEra_ret_20d", "PAYX_Paychex_ret_20d", "Nikkei_Japan_zscore_60d", "MSTR_Bitcoin3_ret_1d"], "is_new": true}, {"model_id": "new_h10_NORMAL_GradientBoosting_N10_t4", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 10, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "CPB_CampbellSoup_zscore_60d", "SPY_zscore_60d", "PFE_ret_1d", "Core_PCE_zscore_60d", "US30Y_Rate_ret_20d", "DHR_vol_20d", "AXP_Amex_ret_20d", "DAX_Germany_vol_20d"], "is_new": true}, {"model_id": "new_h10_NORMAL_GradientBoosting_N10_t5", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 10, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "T_ret_1d", "SBUX_zscore_60d", "IWM_SmallCap_vol_20d", "HangSeng_HK_ret_5d", "INTC_ret_1d", "VRP_ma5", "MSTR_Bitcoin3_ret_1d", "CI_Cigna_vol_20d"], "is_new": true}, {"model_id": "new_h10_NORMAL_GradientBoosting_N10_t6", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 10, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "SCHW_Schwab_ret_5d", "WTI_Oil_FRED_zscore_60d", "EWS_Singapore_ret_5d", "HangSeng_HK_ret_1d", "NWL_Newell_ret_20d", "PPL_PPL_ret_1d", "HangSeng_HK_ret_5d", "T10Y2Y_Spread_ret_5d"], "is_new": true}, {"model_id": "new_h10_NORMAL_GradientBoosting_N10_t7", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 10, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "CLX_Clorox_vol_20d", "EWA_Australia_ret_1d", "MSTR_Bitcoin3_ret_5d", "EWL_Switzerland_zscore_60d", "NVDA_vol_20d", "EMR_Emerson_ret_20d", "Brent_Oil_FRED_ret_5d", "EWJ_Japan_vol_20d"], "is_new": true}, {"model_id": "new_h10_NORMAL_GradientBoosting_N12_t0", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 10, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "CPB_CampbellSoup_ret_20d", "SBUX_zscore_60d", "Retail_Sales_zscore_60d", "EWC_Canada_zscore_60d", "INTC_ret_1d", "JNJ_ret_1d", "LLY_zscore_60d", "NVDA_vol_20d", "EWM_Malaysia_zscore_60d", "MSTR_Bitcoin3_ret_20d"], "is_new": true}, {"model_id": "new_h10_NORMAL_GradientBoosting_N12_t1", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 10, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "US3Y_Rate_ret_5d", "GD_GeneralDynamics_zscore_60d", "EWY_Korea_zscore_60d", "AXP_Amex_vol_20d", "EWA_Australia_zscore_60d", "EWG_Germany_vol_20d", "EWS_Singapore_ret_5d", "NOC_Northrop_ret_20d", "PCAR_PaccarInc_ret_5d", "AVB_AvalonBay_zscore_60d"], "is_new": true}, {"model_id": "new_h10_NORMAL_GradientBoosting_N12_t2", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 10, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "SJM_JM_Smucker_ret_1d", "MS_MorganStanley_zscore_60d", "BTI_BritishAmerican_ret_5d", "BLK_BlackRock_zscore_60d", "EWM_Malaysia_ret_1d", "ITT_ITTInc_ret_5d", "EQR_Equity_ret_1d", "EWA_Australia_ret_1d", "XOM_ret_20d", "EWJ_Japan_vol_20d"], "is_new": true}, {"model_id": "new_h10_NORMAL_GradientBoosting_N12_t3", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 10, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "VRP_ma5", "PLD_Prologis_ret_5d", "3M_ret_5d", "Core_CPI_zscore_60d", "ASX_Australia_ret_5d", "EWY_Korea_ret_20d", "vix_acceleration_1d", "EWQ_France_ret_20d", "SO_SouthernCo_ret_5d", "US3M_Rate_zscore_60d"], "is_new": true}, {"model_id": "new_h10_NORMAL_GradientBoosting_N12_t4", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 10, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWC_Canada_zscore_60d", "M_Macys_vol_20d", "BTI_BritishAmerican_ret_20d", "3M_vol_20d", "CCI_CrownCastle_vol_20d", "TXN_vol_20d", "AXP_Amex_vol_20d", "Core_CPI_zscore_60d", "XLF_Fin_vol_20d", "NWL_Newell_ret_20d"], "is_new": true}, {"model_id": "new_h10_NORMAL_GradientBoosting_N12_t5", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 10, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "US1Y_Rate_ret_20d", "CI_Cigna_vol_20d", "US6M_Rate_ret_20d", "Industrial_Production_zscore_60d", "VOD_Vodafone_zscore_60d", "LUV_SouthwestAir_ret_5d", "CPB_CampbellSoup_zscore_60d", "SPY_zscore_60d", "ORCL_vol_20d", "PAYX_Paychex_zscore_60d"], "is_new": true}, {"model_id": "new_h10_NORMAL_GradientBoosting_N12_t6", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 10, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "US3Y_Rate_ret_5d", "heston_ev_h3", "IYR_US_REIT2_zscore_60d", "CTAS_Cintas_vol_20d", "EWL_Switzerland_vol_20d", "GE_ret_1d", "XLF_Fin_vol_20d", "US30Y_Rate_ret_20d", "JNJ_ret_1d", "XOM_ret_1d"], "is_new": true}, {"model_id": "new_h10_NORMAL_GradientBoosting_N12_t7", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 10, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "DE_Deere_vol_20d", "EWH_HongKong_ret_5d", "PFE_ret_1d", "PAYX_Paychex_vol_20d", "NVDA_vol_20d", "T_ret_1d", "EWL_Switzerland_zscore_60d", "ENB_EnbridgeInc_ret_1d", "Michigan_Sentiment_ret_20d", "TM_Telephone_ret_1d"], "is_new": true}, {"model_id": "new_h10_NORMAL_GradientBoosting_N15_t0", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 10, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "US5Y_Rate_ret_5d", "HD_ret_1d", "MSTR_Bitcoin3_ret_5d", "Core_PCE_zscore_60d", "BTI_BritishAmerican_ret_20d", "AMT_AmericanTower_ret_1d", "MS_MorganStanley_ret_1d", "JNJ_ret_1d", "AMD_ret_1d", "heston_var_ev_h7", "TM_Telephone_ret_1d", "EQIX_Equinix_ret_5d", "LMT_LockheedMartin_ret_1d"], "is_new": true}, {"model_id": "new_h10_NORMAL_GradientBoosting_N15_t1", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 10, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "XLK_Tech_zscore_60d", "NWL_Newell_ret_20d", "PG_ret_20d", "PPL_PPL_ret_1d", "HD_ret_5d", "IYM_BasicMaterials_ret_20d", "NVDA_vol_20d", "ITT_ITTInc_ret_5d", "SLB_Schlumberger_ret_1d", "VOD_Vodafone_zscore_60d", "US3M_Rate_vol_20d", "NFCI_ret_5d", "3M_ret_5d"], "is_new": true}, {"model_id": "new_h10_NORMAL_GradientBoosting_N15_t2", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 10, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWG_Germany_ret_20d", "EWY_Korea_zscore_60d", "US1Y_Rate_ret_20d", "EWL_Switzerland_vol_20d", "PAYX_Paychex_ret_20d", "XOM_ret_1d", "MRK_Merck_zscore_60d", "ENB_EnbridgeInc_ret_1d", "EWM_Malaysia_zscore_60d", "EWA_Australia_zscore_60d", "NWL_Newell_ret_20d", "VRP_ma5", "XLF_Fin_vol_20d"], "is_new": true}, {"model_id": "new_h10_NORMAL_GradientBoosting_N15_t3", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 10, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "MO_AltriaMG_ret_1d", "3M_vol_20d", "EWM_Malaysia_vol_20d", "EWL_Switzerland_zscore_60d", "Industrial_Production_zscore_60d", "SBUX_zscore_60d", "CI_Cigna_vol_20d", "SCHW_Schwab_ret_5d", "PFE_ret_1d", "DIS_vol_20d", "NFCI_ret_5d", "MSTR_Bitcoin3_ret_1d", "XLV_Health_zscore_60d"], "is_new": true}, {"model_id": "new_h10_NORMAL_GradientBoosting_N15_t4", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 10, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EQR_Equity_ret_1d", "BLK_BlackRock_zscore_60d", "JNJ_ret_1d", "XOM_ret_20d", "NVDA_vol_20d", "CTAS_Cintas_vol_20d", "LMT_LockheedMartin_vol_20d", "XLF_Fin_vol_20d", "VOD_Vodafone_zscore_60d", "SJM_JM_Smucker_ret_5d", "vix_acceleration_1d", "ASX_Australia_ret_5d", "heston_var_ev_h5"], "is_new": true}, {"model_id": "new_h10_NORMAL_GradientBoosting_N15_t5", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 10, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWL_Switzerland_zscore_60d", "BDX_Becton_Dickinson_ret_20d", "EWM_Malaysia_zscore_60d", "gjr_condvar_h1", "spx_momentum_3d", "XLY_Disc_vol_20d", "EWH_HongKong_ret_5d", "Brent_Oil_FRED_ret_5d", "CPB_CampbellSoup_ret_20d", "AMZN_ret_5d", "DIS_vol_20d", "MSTR_Bitcoin3_ret_5d", "EWY_Korea_zscore_60d"], "is_new": true}, {"model_id": "new_h10_NORMAL_GradientBoosting_N15_t6", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 10, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "TGT_Target_zscore_60d", "DHR_vol_20d", "LMT_LockheedMartin_ret_1d", "EQR_Equity_ret_1d", "EMR_Emerson_ret_20d", "XLB_Materials_zscore_60d", "US3M_Rate_vol_20d", "BTI_BritishAmerican_ret_5d", "Core_CPI_zscore_60d", "vix_acceleration_1d", "GD_GeneralDynamics_zscore_60d", "BA_ret_1d", "3M_vol_20d"], "is_new": true}, {"model_id": "new_h10_NORMAL_GradientBoosting_N15_t7", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 10, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "SCHW_Schwab_ret_5d", "GD_GeneralDynamics_zscore_60d", "EWA_Australia_zscore_60d", "HD_ret_5d", "ORCL_vol_20d", "GE_ret_1d", "TM_Telephone_ret_1d", "hmm_p_stress", "DAX_Germany_vol_20d", "SLB_Schlumberger_ret_1d", "Michigan_Sentiment_ret_20d", "TED_Spread_vol_20d", "EXC_Exelon_ret_1d"], "is_new": true}, {"model_id": "new_h10_NORMAL_GradientBoosting_N20_t0", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 10, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "PAYX_Paychex_ret_20d", "US6M_Rate_ret_20d", "VOD_Vodafone_zscore_60d", "PG_ret_20d", "EWQ_France_ret_20d", "TM_Telephone_ret_1d", "US1Y_Rate_ret_20d", "AMT_AmericanTower_ret_1d", "LMT_LockheedMartin_vol_20d", "EOG_EOGResources_ret_5d", "JNJ_ret_1d", "3M_vol_20d", "XLV_Health_zscore_60d", "NEE_NextEra_ret_20d", "LMT_LockheedMartin_ret_1d", "MS_MorganStanley_zscore_60d", "TM_Telephone_vol_20d", "MSTR_Bitcoin3_ret_20d"], "is_new": true}, {"model_id": "new_h10_NORMAL_GradientBoosting_N20_t1", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 10, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "HD_zscore_60d", "PAYX_Paychex_ret_20d", "EOG_EOGResources_ret_5d", "EWL_Switzerland_zscore_60d", "ENB_EnbridgeInc_ret_1d", "SPY_zscore_60d", "TGT_Target_zscore_60d", "PAYX_Paychex_zscore_60d", "PAYX_Paychex_vol_20d", "NWL_Newell_ret_20d", "US5Y_Rate_ret_5d", "EWM_Malaysia_zscore_60d", "VOD_Vodafone_zscore_60d", "XOM_ret_1d", "BDX_Becton_Dickinson_ret_20d", "MS_MorganStanley_ret_5d", "DE_Deere_vol_20d", "BTI_BritishAmerican_ret_20d"], "is_new": true}, {"model_id": "new_h10_NORMAL_GradientBoosting_N20_t2", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 10, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "Brent_Oil_FRED_ret_5d", "PCAR_PaccarInc_ret_5d", "3M_vol_20d", "US3M_Rate_zscore_60d", "DAX_Germany_vol_20d", "LOW_Lowes_ret_5d", "US5Y_Rate_ret_5d", "vix_mean_abs_ret_5d", "EWL_Switzerland_vol_20d", "BTI_BritishAmerican_ret_20d", "MS_MorganStanley_ret_1d", "HD_zscore_60d", "XLB_Materials_zscore_60d", "heston_var_ev_h3", "EWA_Australia_zscore_60d", "PG_ret_20d", "TM_Telephone_vol_20d", "BLK_BlackRock_zscore_60d"], "is_new": true}, {"model_id": "new_h10_NORMAL_GradientBoosting_N20_t3", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 10, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "WTI_Oil_FRED_zscore_60d", "BA_ret_1d", "spx_abs_ret_max_5d", "AORD_AUS_zscore_60d", "TED_Spread_zscore_60d", "gjr_condvar_h1", "US7Y_Rate_ret_20d", "EXC_Exelon_ret_1d", "BTI_BritishAmerican_ret_5d", "IWM_SmallCap_vol_20d", "SCHW_Schwab_ret_5d", "ES_Evergy_ret_1d", "DHR_vol_20d", "CPB_CampbellSoup_ret_20d", "XLY_Disc_vol_20d", "NVDA_vol_20d", "XLV_Health_zscore_60d", "EWA_Australia_zscore_60d"], "is_new": true}, {"model_id": "new_h10_NORMAL_GradientBoosting_N20_t4", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 10, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "DAX_Germany_vol_20d", "AMZN_ret_5d", "EMR_Emerson_ret_20d", "LLY_zscore_60d", "CLX_Clorox_vol_20d", "T10Y2Y_Spread_ret_5d", "INTC_ret_5d", "SPY_zscore_60d", "EWM_Malaysia_vol_20d", "QQQ_vol_20d", "XLB_Materials_zscore_60d", "CPB_CampbellSoup_ret_5d", "US1Y_Rate_ret_20d", "LUV_SouthwestAir_ret_5d", "US30Y_Rate_ret_20d", "EXC_Exelon_zscore_60d", "Nikkei_Japan_zscore_60d", "PFE_ret_1d"], "is_new": true}, {"model_id": "new_h10_NORMAL_GradientBoosting_N20_t5", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 10, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EFFR_vol_20d", "CCI_CrownCastle_vol_20d", "gjr_condvar_h1", "PAYX_Paychex_zscore_60d", "heston_var_ev_h3", "XLB_Materials_zscore_60d", "SLB_Schlumberger_ret_1d", "EFFR_ret_1d", "JNJ_ret_1d", "AXP_Amex_ret_20d", "PG_ret_20d", "IYM_BasicMaterials_ret_20d", "EWL_Switzerland_zscore_60d", "ENB_EnbridgeInc_ret_1d", "AMD_ret_1d", "vix_acceleration_1d", "TGT_Target_zscore_60d", "INTC_ret_1d"], "is_new": true}, {"model_id": "new_h10_NORMAL_GradientBoosting_N20_t6", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 10, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "GD_GeneralDynamics_zscore_60d", "AXP_Amex_vol_20d", "Core_PCE_zscore_60d", "spx_momentum_3d", "EWC_Canada_zscore_60d", "EWM_Malaysia_zscore_60d", "LOW_Lowes_ret_20d", "PG_ret_20d", "HD_ret_20d", "heston_var_ev_h7", "NFCI_ret_5d", "IYM_BasicMaterials_ret_20d", "EWH_HongKong_ret_5d", "NVDA_vol_20d", "MO_AltriaMG_ret_1d", "EFFR_vol_20d", "JNJ_ret_1d", "EWS_Singapore_ret_5d"], "is_new": true}, {"model_id": "new_h10_NORMAL_GradientBoosting_N20_t7", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 10, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "GD_GeneralDynamics_zscore_60d", "vix_acceleration_1d", "IYR_US_REIT2_zscore_60d", "SLB_Schlumberger_ret_1d", "AMD_ret_5d", "HD_ret_20d", "PCAR_PaccarInc_ret_5d", "AORD_AUS_zscore_60d", "US5Y_Rate_ret_5d", "EFFR_vol_20d", "TM_Telephone_vol_20d", "US6M_Rate_ret_20d", "EWH_HongKong_ret_5d", "AXP_Amex_vol_20d", "INTC_ret_5d", "US7Y_Rate_ret_20d", "XLK_Tech_zscore_60d", "PAYX_Paychex_ret_20d"], "is_new": true}, {"model_id": "new_h10_NORMAL_GradientBoosting_N25_t0", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 10, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "DE_Deere_ret_5d", "Brent_Oil_FRED_ret_5d", "EOG_EOGResources_ret_5d", "SJM_JM_Smucker_ret_5d", "M_Macys_vol_20d", "US7Y_Rate_ret_20d", "TXN_vol_20d", "EFFR_ret_1d", "AVB_AvalonBay_zscore_60d", "spx_vol_5d", "LOW_Lowes_ret_5d", "HD_zscore_60d", "EMR_Emerson_ret_20d", "AXP_Amex_ret_20d", "AMGN_Amgen_ret_1d", "EWM_Malaysia_ret_1d", "CI_Cigna_vol_20d", "JNJ_ret_1d", "AMT_AmericanTower_ret_1d", "SCHW_Schwab_ret_5d", "spx_momentum_3d", "ASX_Australia_ret_5d", "CLX_Clorox_vol_20d"], "is_new": true}, {"model_id": "new_h10_NORMAL_GradientBoosting_N25_t1", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 10, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWY_Korea_zscore_60d", "spx_momentum_3d", "Core_PCE_zscore_60d", "EWG_Germany_vol_20d", "CCI_CrownCastle_vol_20d", "TXN_vol_20d", "GILD_Gilead_ret_20d", "SJM_JM_Smucker_ret_5d", "ASX_Australia_vol_20d", "EWG_Germany_ret_20d", "DE_Deere_ret_5d", "SLB_Schlumberger_ret_1d", "DIS_vol_20d", "DAX_Germany_vol_20d", "HD_ret_20d", "DAX_Germany_zscore_60d", "SBUX_vol_20d", "XLF_Fin_vol_20d", "hmm_p_stress", "spx_abs_ret_max_5d", "ASX_Australia_ret_5d", "heston_var_ev_h5", "XLV_Health_zscore_60d"], "is_new": true}, {"model_id": "new_h10_NORMAL_GradientBoosting_N25_t2", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 10, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "CPB_CampbellSoup_zscore_60d", "MSTR_Bitcoin3_ret_5d", "TM_Telephone_vol_20d", "EWJ_Japan_vol_20d", "CPB_CampbellSoup_ret_5d", "AMGN_Amgen_ret_1d", "T_ret_1d", "HUM_Humana_ret_5d", "MS_MorganStanley_zscore_60d", "LLY_zscore_60d", "MSTR_Bitcoin3_ret_20d", "FedFunds_zscore_60d", "EQR_Equity_ret_1d", "EWA_Australia_zscore_60d", "EQIX_Equinix_ret_5d", "IBEX_Spain_ret_20d", "MO_AltriaMG_ret_1d", "EWY_Korea_zscore_60d", "ITT_ITTInc_ret_5d", "EXC_Exelon_ret_1d", "CLX_Clorox_vol_20d", "CCI_CrownCastle_vol_20d", "EWH_HongKong_ret_5d"], "is_new": true}, {"model_id": "new_h10_NORMAL_GradientBoosting_N25_t3", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 10, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "ES_Evergy_ret_1d", "vix_mean_abs_ret_5d", "PAYX_Paychex_ret_20d", "DHR_vol_20d", "M_Macys_vol_20d", "CTAS_Cintas_vol_20d", "T_ret_1d", "EWS_Singapore_ret_5d", "spx_momentum_3d", "TED_Spread_vol_20d", "HangSeng_HK_vol_20d", "US1Y_Rate_ret_20d", "US3M_Rate_vol_20d", "EWG_Germany_ret_20d", "LMT_LockheedMartin_vol_20d", "XOM_ret_1d", "US7Y_Rate_ret_20d", "SJM_JM_Smucker_ret_5d", "MRK_Merck_zscore_60d", "hmm_p_stress", "BLK_BlackRock_zscore_60d", "CMCSA_ret_1d", "PCAR_PaccarInc_ret_5d"], "is_new": true}, {"model_id": "new_h10_NORMAL_GradientBoosting_N25_t4", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 10, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "Nikkei_Japan_zscore_60d", "PAYX_Paychex_vol_20d", "DHR_vol_20d", "MS_MorganStanley_zscore_60d", "INTC_ret_1d", "PPL_PPL_ret_1d", "XLB_Materials_zscore_60d", "EWA_Australia_ret_1d", "SJM_JM_Smucker_ret_1d", "EQIX_Equinix_ret_5d", "CPB_CampbellSoup_vol_20d", "US1Y_Rate_ret_20d", "Retail_Sales_zscore_60d", "SBUX_vol_20d", "EWY_Korea_ret_20d", "LMT_LockheedMartin_ret_1d", "LOW_Lowes_ret_5d", "AMD_ret_1d", "SLB_Schlumberger_ret_1d", "GILD_Gilead_ret_20d", "EWM_Malaysia_ret_1d", "MSTR_Bitcoin3_ret_20d", "HD_ret_5d"], "is_new": true}, {"model_id": "new_h10_NORMAL_GradientBoosting_N25_t5", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 10, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "Core_CPI_zscore_60d", "Retail_Sales_zscore_60d", "HangSeng_HK_vol_20d", "3M_vol_20d", "US3M_Rate_vol_20d", "MSTR_Bitcoin3_ret_20d", "MSTR_Bitcoin3_ret_1d", "NWL_Newell_ret_20d", "EQR_Equity_ret_1d", "EFFR_ret_1d", "EWY_Korea_ret_20d", "GD_GeneralDynamics_zscore_60d", "NOC_Northrop_ret_20d", "DAX_Germany_zscore_60d", "US3Y_Rate_ret_5d", "PAYX_Paychex_zscore_60d", "VVIX_ret_20d", "CTAS_Cintas_vol_20d", "IYM_BasicMaterials_ret_20d", "Core_PCE_zscore_60d", "FedFunds_zscore_60d", "EWA_Australia_zscore_60d", "AMZN_ret_5d"], "is_new": true}, {"model_id": "new_h10_NORMAL_GradientBoosting_N25_t6", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 10, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "XOM_ret_20d", "T_ret_1d", "vix_acceleration_1d", "NOC_Northrop_ret_20d", "3M_ret_5d", "3M_vol_20d", "XLK_Tech_zscore_60d", "EWH_HongKong_ret_5d", "XLF_Fin_vol_20d", "CCI_CrownCastle_vol_20d", "vix_mean_abs_ret_5d", "Nikkei_Japan_vol_20d", "DAX_Germany_vol_20d", "SCHW_Schwab_ret_5d", "TM_Telephone_vol_20d", "gjr_condvar_h1", "CPB_CampbellSoup_vol_20d", "ASX_Australia_vol_20d", "spx_momentum_3d", "EWY_Korea_zscore_60d", "VRP_ma5", "LOW_Lowes_ret_20d", "HangSeng_HK_ret_5d"], "is_new": true}, {"model_id": "new_h10_NORMAL_GradientBoosting_N25_t7", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 10, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWG_Germany_ret_20d", "spx_momentum_3d", "T10Y2Y_Spread_ret_5d", "BTI_BritishAmerican_ret_5d", "DE_Deere_vol_20d", "VVIX_ret_20d", "BA_ret_1d", "FedFunds_zscore_60d", "DHR_ret_1d", "CPB_CampbellSoup_ret_5d", "HD_ret_20d", "BDX_Becton_Dickinson_ret_20d", "TED_Spread_vol_20d", "AMT_AmericanTower_ret_1d", "heston_ev_h3", "PFE_ret_1d", "IBEX_Spain_ret_20d", "SO_SouthernCo_ret_5d", "EWH_HongKong_ret_5d", "XOM_ret_1d", "EWL_Switzerland_vol_20d", "NWL_Newell_ret_20d", "CLX_Clorox_vol_20d"], "is_new": true}, {"model_id": "new_h10_NORMAL_GradientBoosting_N30_t0", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 10, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "PFE_ret_1d", "heston_var_ev_h5", "HD_ret_5d", "XOM_ret_20d", "T10Y2Y_Spread_ret_5d", "BLK_BlackRock_zscore_60d", "SCHW_Schwab_ret_5d", "EWL_Switzerland_vol_20d", "3M_ret_5d", "HD_ret_20d", "NOC_Northrop_ret_20d", "EWL_Switzerland_zscore_60d", "Retail_Sales_zscore_60d", "HangSeng_HK_vol_20d", "vix_mean_abs_ret_5d", "LLY_zscore_60d", "EWA_Australia_zscore_60d", "FedFunds_zscore_60d", "EWG_Germany_ret_20d", "ITT_ITTInc_ret_5d", "SJM_JM_Smucker_ret_1d", "NFCI_ret_5d", "3M_vol_20d", "MRK_Merck_zscore_60d", "EXC_Exelon_zscore_60d", "BDX_Becton_Dickinson_ret_20d", "CLX_Clorox_vol_20d", "US6M_Rate_ret_20d"], "is_new": true}, {"model_id": "new_h10_NORMAL_GradientBoosting_N30_t1", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 10, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "VVIX_ret_20d", "EWY_Korea_zscore_60d", "EWA_Australia_ret_1d", "EWA_Australia_zscore_60d", "T10Y2Y_Spread_ret_5d", "HD_ret_1d", "Core_PCE_zscore_60d", "US7Y_Rate_ret_20d", "HangSeng_HK_ret_5d", "US6M_Rate_ret_20d", "ITT_ITTInc_ret_5d", "NFCI_ret_5d", "EWL_Switzerland_vol_20d", "SBUX_zscore_60d", "T_ret_1d", "BDX_Becton_Dickinson_ret_20d", "PFE_ret_1d", "HD_ret_20d", "LUV_SouthwestAir_ret_5d", "BTI_BritishAmerican_ret_5d", "LLY_zscore_60d", "US30Y_Rate_ret_20d", "M_Macys_vol_20d", "IYM_BasicMaterials_ret_20d", "PAYX_Paychex_ret_20d", "EWM_Malaysia_zscore_60d", "CCI_CrownCastle_vol_20d", "PG_ret_20d"], "is_new": true}, {"model_id": "new_h10_NORMAL_GradientBoosting_N30_t2", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 10, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "ORCL_zscore_60d", "NWL_Newell_ret_20d", "DIS_vol_20d", "CPB_CampbellSoup_ret_5d", "EOG_EOGResources_ret_5d", "EQIX_Equinix_ret_5d", "US1Y_Rate_ret_5d", "MS_MorganStanley_ret_1d", "Michigan_Sentiment_ret_20d", "JNJ_ret_1d", "Brent_Oil_FRED_ret_5d", "Nikkei_Japan_zscore_60d", "CPB_CampbellSoup_ret_20d", "NFCI_ret_5d", "XLY_Disc_vol_20d", "BDX_Becton_Dickinson_ret_20d", "CLX_Clorox_vol_20d", "WTI_Oil_FRED_zscore_60d", "EWA_Australia_zscore_60d", "BLK_BlackRock_zscore_60d", "heston_var_ev_h5", "EWQ_France_ret_20d", "MS_MorganStanley_ret_5d", "AMGN_Amgen_ret_1d", "EWA_Australia_ret_1d", "AXP_Amex_vol_20d", "GILD_Gilead_ret_20d", "TED_Spread_vol_20d"], "is_new": true}, {"model_id": "new_h10_NORMAL_GradientBoosting_N30_t3", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 10, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWL_Switzerland_zscore_60d", "MSTR_Bitcoin3_ret_20d", "TED_Spread_zscore_60d", "heston_ev_h3", "VRP_ma5", "WTI_Oil_FRED_zscore_60d", "CMCSA_ret_1d", "TGT_Target_zscore_60d", "US3Y_Rate_ret_5d", "ENB_EnbridgeInc_ret_1d", "gjr_condvar_h1", "PPL_PPL_ret_1d", "SBUX_vol_20d", "LMT_LockheedMartin_vol_20d", "ORCL_zscore_60d", "spx_abs_ret_max_5d", "AMD_ret_5d", "vix_acceleration_1d", "3M_vol_20d", "EWA_Australia_zscore_60d", "DIS_vol_20d", "AMGN_Amgen_ret_1d", "US3M_Rate_zscore_60d", "CLX_Clorox_vol_20d", "Industrial_Production_zscore_60d", "EFFR_vol_20d", "CCI_CrownCastle_vol_20d", "IYM_BasicMaterials_ret_20d"], "is_new": true}, {"model_id": "new_h10_NORMAL_GradientBoosting_N30_t4", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 10, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "MSTR_Bitcoin3_ret_20d", "ASX_Australia_vol_20d", "US7Y_Rate_ret_20d", "DHR_vol_20d", "EWQ_France_ret_20d", "TED_Spread_zscore_60d", "INTC_ret_1d", "LMT_LockheedMartin_vol_20d", "EWA_Australia_zscore_60d", "EQR_Equity_ret_1d", "TM_Telephone_vol_20d", "EMR_Emerson_ret_20d", "heston_ev_h3", "SJM_JM_Smucker_ret_5d", "vix_mean_abs_ret_5d", "EXC_Exelon_ret_1d", "FedFunds_zscore_60d", "CTAS_Cintas_vol_20d", "EXC_Exelon_zscore_60d", "CPB_CampbellSoup_ret_5d", "US1Y_Rate_ret_20d", "EWG_Germany_ret_20d", "US3M_Rate_zscore_60d", "XLF_Fin_vol_20d", "Michigan_Sentiment_ret_20d", "Nikkei_Japan_zscore_60d", "IYR_US_REIT2_zscore_60d", "AMZN_ret_5d"], "is_new": true}, {"model_id": "new_h10_NORMAL_GradientBoosting_N30_t5", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 10, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "heston_var_ev_h7", "EWM_Malaysia_ret_1d", "MSTR_Bitcoin3_ret_5d", "AXP_Amex_ret_20d", "MSTR_Bitcoin3_ret_20d", "HangSeng_HK_ret_5d", "EWA_Australia_ret_1d", "T_ret_1d", "ASX_Australia_ret_5d", "VVIX_ret_20d", "EWA_Australia_zscore_60d", "DE_Deere_ret_5d", "ORCL_zscore_60d", "IBEX_Spain_ret_20d", "EWH_HongKong_ret_5d", "LOW_Lowes_ret_5d", "AMZN_ret_5d", "TXN_vol_20d", "EWG_Germany_vol_20d", "HD_zscore_60d", "AMT_AmericanTower_ret_1d", "AMD_ret_1d", "EWM_Malaysia_vol_20d", "XLF_Fin_vol_20d", "TM_Telephone_vol_20d", "EWY_Korea_ret_20d", "VOD_Vodafone_zscore_60d", "BLK_BlackRock_zscore_60d"], "is_new": true}, {"model_id": "new_h10_NORMAL_GradientBoosting_N30_t6", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 10, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "DE_Deere_vol_20d", "IYR_US_REIT2_zscore_60d", "TED_Spread_vol_20d", "XOM_ret_1d", "EWY_Korea_zscore_60d", "FedFunds_zscore_60d", "HangSeng_HK_ret_1d", "Industrial_Production_zscore_60d", "IWM_SmallCap_vol_20d", "EWG_Germany_ret_20d", "CMCSA_ret_1d", "XLY_Disc_vol_20d", "AXP_Amex_ret_20d", "MO_AltriaMG_ret_1d", "LMT_LockheedMartin_ret_1d", "XLK_Tech_zscore_60d", "TM_Telephone_ret_1d", "CI_Cigna_vol_20d", "SJM_JM_Smucker_ret_5d", "SLB_Schlumberger_ret_1d", "LOW_Lowes_ret_20d", "EWY_Korea_ret_20d", "IBEX_Spain_ret_20d", "EXC_Exelon_ret_1d", "HD_zscore_60d", "US30Y_Rate_ret_20d", "MRK_Merck_zscore_60d", "AMZN_ret_5d"], "is_new": true}, {"model_id": "new_h10_NORMAL_GradientBoosting_N30_t7", "algo": "GradientBoosting", "regime": "NORMAL", "horizon": 10, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "heston_ev_h3", "GILD_Gilead_ret_20d", "US3Y_Rate_ret_5d", "LMT_LockheedMartin_ret_1d", "US3M_Rate_zscore_60d", "Brent_Oil_FRED_ret_20d", "NEE_NextEra_ret_20d", "US6M_Rate_ret_20d", "ORCL_vol_20d", "AMD_ret_1d", "SLB_Schlumberger_ret_1d", "HangSeng_HK_vol_20d", "CPB_CampbellSoup_ret_5d", "hmm_p_stress", "CPB_CampbellSoup_vol_20d", "HangSeng_HK_ret_1d", "HD_ret_1d", "BA_ret_1d", "QQQ_vol_20d", "IBEX_Spain_ret_20d", "AORD_AUS_zscore_60d", "IYM_BasicMaterials_ret_20d", "EOG_EOGResources_vol_20d", "DOW_Price_zscore_60d", "LOW_Lowes_ret_20d", "ORCL_zscore_60d", "DE_Deere_ret_5d", "US5Y_Rate_ret_5d"], "is_new": true}, {"model_id": "new_h10_NORMAL_RandomForest_N5_t0", "algo": "RandomForest", "regime": "NORMAL", "horizon": 10, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "AMD_ret_1d", "AMGN_Amgen_ret_1d", "EWH_HongKong_ret_5d"], "is_new": true}, {"model_id": "new_h10_NORMAL_RandomForest_N5_t1", "algo": "RandomForest", "regime": "NORMAL", "horizon": 10, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "US5Y_Rate_ret_5d", "XLB_Materials_zscore_60d", "HD_zscore_60d"], "is_new": true}, {"model_id": "new_h10_NORMAL_RandomForest_N5_t2", "algo": "RandomForest", "regime": "NORMAL", "horizon": 10, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "ITT_ITTInc_ret_5d", "EWM_Malaysia_vol_20d", "DIS_vol_20d"], "is_new": true}, {"model_id": "new_h10_NORMAL_RandomForest_N5_t3", "algo": "RandomForest", "regime": "NORMAL", "horizon": 10, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "BTI_BritishAmerican_ret_20d", "EQR_Equity_ret_1d", "PG_ret_20d"], "is_new": true}, {"model_id": "new_h10_NORMAL_RandomForest_N5_t4", "algo": "RandomForest", "regime": "NORMAL", "horizon": 10, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "Nikkei_Japan_zscore_60d", "XLK_Tech_zscore_60d", "LOW_Lowes_ret_20d"], "is_new": true}, {"model_id": "new_h10_NORMAL_RandomForest_N5_t5", "algo": "RandomForest", "regime": "NORMAL", "horizon": 10, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWH_HongKong_ret_5d", "INTC_ret_5d", "US5Y_Rate_ret_5d"], "is_new": true}, {"model_id": "new_h10_NORMAL_RandomForest_N5_t6", "algo": "RandomForest", "regime": "NORMAL", "horizon": 10, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "Brent_Oil_FRED_ret_5d", "LLY_zscore_60d", "CCI_CrownCastle_vol_20d"], "is_new": true}, {"model_id": "new_h10_NORMAL_RandomForest_N5_t7", "algo": "RandomForest", "regime": "NORMAL", "horizon": 10, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "DIS_vol_20d", "DHR_ret_1d", "US30Y_Rate_ret_20d"], "is_new": true}, {"model_id": "new_h10_NORMAL_RandomForest_N8_t0", "algo": "RandomForest", "regime": "NORMAL", "horizon": 10, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "SBUX_ret_5d", "GILD_Gilead_ret_20d", "TM_Telephone_ret_1d", "EWJ_Japan_vol_20d", "Retail_Sales_zscore_60d", "vix_mean_abs_ret_5d"], "is_new": true}, {"model_id": "new_h10_NORMAL_RandomForest_N8_t1", "algo": "RandomForest", "regime": "NORMAL", "horizon": 10, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWQ_France_zscore_60d", "ES_Evergy_ret_1d", "HD_ret_1d", "Retail_Sales_zscore_60d", "CMCSA_ret_1d", "PAYX_Paychex_zscore_60d"], "is_new": true}, {"model_id": "new_h10_NORMAL_RandomForest_N8_t2", "algo": "RandomForest", "regime": "NORMAL", "horizon": 10, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "TM_Telephone_ret_1d", "NVDA_vol_20d", "ASX_Australia_ret_5d", "SPY_zscore_60d", "US1Y_Rate_ret_20d", "EQR_Equity_ret_1d"], "is_new": true}, {"model_id": "new_h10_NORMAL_RandomForest_N8_t3", "algo": "RandomForest", "regime": "NORMAL", "horizon": 10, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "NEE_NextEra_ret_20d", "SCHW_Schwab_ret_5d", "AXP_Amex_vol_20d", "VRP_ma5", "EMR_Emerson_ret_20d", "LOW_Lowes_ret_20d"], "is_new": true}, {"model_id": "new_h10_NORMAL_RandomForest_N8_t4", "algo": "RandomForest", "regime": "NORMAL", "horizon": 10, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "HangSeng_HK_vol_20d", "SJM_JM_Smucker_ret_1d", "INTC_ret_5d", "TED_Spread_zscore_60d", "WTI_Oil_FRED_zscore_60d", "CMCSA_ret_1d"], "is_new": true}, {"model_id": "new_h10_NORMAL_RandomForest_N8_t5", "algo": "RandomForest", "regime": "NORMAL", "horizon": 10, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWC_Canada_zscore_60d", "heston_var_ev_h5", "SJM_JM_Smucker_ret_1d", "CCI_CrownCastle_vol_20d", "SLB_Schlumberger_ret_5d", "EWL_Switzerland_zscore_60d"], "is_new": true}, {"model_id": "new_h10_NORMAL_RandomForest_N8_t6", "algo": "RandomForest", "regime": "NORMAL", "horizon": 10, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "US3M_Rate_zscore_60d", "TGT_Target_zscore_60d", "PPL_PPL_ret_1d", "NOC_Northrop_ret_20d", "VOD_Vodafone_zscore_60d", "DAX_Germany_vol_20d"], "is_new": true}, {"model_id": "new_h10_NORMAL_RandomForest_N8_t7", "algo": "RandomForest", "regime": "NORMAL", "horizon": 10, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "MSTR_Bitcoin3_ret_5d", "MO_AltriaMG_ret_1d", "EOG_EOGResources_vol_20d", "ORCL_zscore_60d", "WTI_Oil_FRED_zscore_60d", "ORCL_vol_20d"], "is_new": true}, {"model_id": "new_h10_NORMAL_RandomForest_N10_t0", "algo": "RandomForest", "regime": "NORMAL", "horizon": 10, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "DIS_vol_20d", "AMD_ret_1d", "EWC_Canada_zscore_60d", "XLB_Materials_zscore_60d", "DHR_ret_1d", "CMCSA_ret_1d", "BDX_Becton_Dickinson_ret_20d", "CPB_CampbellSoup_ret_5d"], "is_new": true}, {"model_id": "new_h10_NORMAL_RandomForest_N10_t1", "algo": "RandomForest", "regime": "NORMAL", "horizon": 10, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "SJM_JM_Smucker_ret_5d", "CI_Cigna_vol_20d", "DAX_Germany_vol_20d", "PPL_PPL_ret_1d", "M_Macys_vol_20d", "US3M_Rate_vol_20d", "EXC_Exelon_zscore_60d", "INTC_ret_1d"], "is_new": true}, {"model_id": "new_h10_NORMAL_RandomForest_N10_t2", "algo": "RandomForest", "regime": "NORMAL", "horizon": 10, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "spx_vol_5d", "EWG_Germany_ret_20d", "AMD_ret_5d", "XLK_Tech_zscore_60d", "NVDA_vol_20d", "MRK_Merck_zscore_60d", "AVB_AvalonBay_zscore_60d", "ASX_Australia_vol_20d"], "is_new": true}, {"model_id": "new_h10_NORMAL_RandomForest_N10_t3", "algo": "RandomForest", "regime": "NORMAL", "horizon": 10, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "ORCL_vol_20d", "HUM_Humana_ret_5d", "LMT_LockheedMartin_ret_1d", "TED_Spread_vol_20d", "CPB_CampbellSoup_ret_20d", "PAYX_Paychex_zscore_60d", "gjr_condvar_h1", "XLY_Disc_vol_20d"], "is_new": true}, {"model_id": "new_h10_NORMAL_RandomForest_N10_t4", "algo": "RandomForest", "regime": "NORMAL", "horizon": 10, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "DAX_Germany_vol_20d", "GILD_Gilead_ret_20d", "AMGN_Amgen_ret_1d", "NWL_Newell_ret_20d", "CI_Cigna_vol_20d", "EFFR_ret_1d", "EWG_Germany_vol_20d", "gjr_condvar_h1"], "is_new": true}, {"model_id": "new_h10_NORMAL_RandomForest_N10_t5", "algo": "RandomForest", "regime": "NORMAL", "horizon": 10, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "CLX_Clorox_vol_20d", "EWQ_France_ret_20d", "AMT_AmericanTower_ret_1d", "EWQ_France_zscore_60d", "SPY_zscore_60d", "US6M_Rate_ret_20d", "Retail_Sales_zscore_60d", "PAYX_Paychex_ret_20d"], "is_new": true}, {"model_id": "new_h10_NORMAL_RandomForest_N10_t6", "algo": "RandomForest", "regime": "NORMAL", "horizon": 10, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "LMT_LockheedMartin_ret_1d", "AMZN_ret_5d", "BDX_Becton_Dickinson_ret_20d", "PAYX_Paychex_zscore_60d", "IYM_BasicMaterials_ret_20d", "HUM_Humana_ret_5d", "AVB_AvalonBay_zscore_60d", "NOC_Northrop_ret_20d"], "is_new": true}, {"model_id": "new_h10_NORMAL_RandomForest_N10_t7", "algo": "RandomForest", "regime": "NORMAL", "horizon": 10, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EFFR_ret_1d", "IWM_SmallCap_vol_20d", "LLY_zscore_60d", "XOM_ret_20d", "IYR_US_REIT2_zscore_60d", "CPB_CampbellSoup_ret_20d", "DAX_Germany_vol_20d", "US5Y_Rate_ret_5d"], "is_new": true}, {"model_id": "new_h10_NORMAL_RandomForest_N12_t0", "algo": "RandomForest", "regime": "NORMAL", "horizon": 10, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "CPB_CampbellSoup_ret_5d", "ORCL_zscore_60d", "vix_acceleration_1d", "EOG_EOGResources_vol_20d", "PAYX_Paychex_ret_20d", "DHR_ret_1d", "AMD_ret_5d", "MS_MorganStanley_ret_1d", "LUV_SouthwestAir_ret_5d", "heston_var_ev_h5"], "is_new": true}, {"model_id": "new_h10_NORMAL_RandomForest_N12_t1", "algo": "RandomForest", "regime": "NORMAL", "horizon": 10, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWH_HongKong_ret_5d", "SLB_Schlumberger_ret_1d", "NEE_NextEra_ret_20d", "EWA_Australia_zscore_60d", "NWL_Newell_ret_20d", "DHR_vol_20d", "BA_ret_1d", "EFFR_vol_20d", "Nikkei_Japan_zscore_60d", "CPB_CampbellSoup_ret_20d"], "is_new": true}, {"model_id": "new_h10_NORMAL_RandomForest_N12_t2", "algo": "RandomForest", "regime": "NORMAL", "horizon": 10, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "ITT_ITTInc_ret_5d", "EQR_Equity_ret_1d", "spx_momentum_3d", "TXN_vol_20d", "EWA_Australia_ret_1d", "CTAS_Cintas_vol_20d", "US1Y_Rate_ret_20d", "PAYX_Paychex_ret_20d", "ASX_Australia_ret_5d", "HD_ret_5d"], "is_new": true}, {"model_id": "new_h10_NORMAL_RandomForest_N12_t3", "algo": "RandomForest", "regime": "NORMAL", "horizon": 10, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "HD_zscore_60d", "BA_ret_1d", "EWM_Malaysia_vol_20d", "EWY_Korea_zscore_60d", "Retail_Sales_zscore_60d", "MSTR_Bitcoin3_ret_1d", "EWL_Switzerland_zscore_60d", "EWA_Australia_zscore_60d", "HangSeng_HK_ret_5d", "PCAR_PaccarInc_ret_5d"], "is_new": true}, {"model_id": "new_h10_NORMAL_RandomForest_N12_t4", "algo": "RandomForest", "regime": "NORMAL", "horizon": 10, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWL_Switzerland_zscore_60d", "BTI_BritishAmerican_ret_5d", "ORCL_vol_20d", "XLB_Materials_zscore_60d", "NEE_NextEra_ret_20d", "LUV_SouthwestAir_ret_5d", "AMZN_ret_5d", "EWM_Malaysia_vol_20d", "US30Y_Rate_ret_20d", "EOG_EOGResources_vol_20d"], "is_new": true}, {"model_id": "new_h10_NORMAL_RandomForest_N12_t5", "algo": "RandomForest", "regime": "NORMAL", "horizon": 10, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "SJM_JM_Smucker_ret_1d", "BDX_Becton_Dickinson_ret_20d", "AMD_ret_1d", "vix_mean_abs_ret_5d", "AXP_Amex_ret_20d", "TM_Telephone_vol_20d", "MS_MorganStanley_zscore_60d", "PG_ret_20d", "Michigan_Sentiment_ret_20d", "IBEX_Spain_ret_20d"], "is_new": true}, {"model_id": "new_h10_NORMAL_RandomForest_N12_t6", "algo": "RandomForest", "regime": "NORMAL", "horizon": 10, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "INTC_ret_1d", "US3M_Rate_zscore_60d", "LLY_zscore_60d", "Core_PCE_zscore_60d", "CPB_CampbellSoup_zscore_60d", "Brent_Oil_FRED_ret_5d", "AMD_ret_5d", "Michigan_Sentiment_ret_20d", "TM_Telephone_ret_1d", "spx_abs_ret_max_5d"], "is_new": true}, {"model_id": "new_h10_NORMAL_RandomForest_N12_t7", "algo": "RandomForest", "regime": "NORMAL", "horizon": 10, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "IYM_BasicMaterials_ret_20d", "US3Y_Rate_ret_5d", "EWY_Korea_ret_20d", "AXP_Amex_vol_20d", "PPL_PPL_ret_1d", "SLB_Schlumberger_ret_5d", "EWM_Malaysia_zscore_60d", "AMZN_ret_5d", "CPB_CampbellSoup_zscore_60d", "Michigan_Sentiment_ret_20d"], "is_new": true}, {"model_id": "new_h10_NORMAL_RandomForest_N15_t0", "algo": "RandomForest", "regime": "NORMAL", "horizon": 10, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "GILD_Gilead_ret_20d", "HangSeng_HK_ret_5d", "TM_Telephone_ret_1d", "EWJ_Japan_vol_20d", "AMGN_Amgen_ret_1d", "PG_ret_20d", "LMT_LockheedMartin_vol_20d", "LOW_Lowes_ret_20d", "TED_Spread_zscore_60d", "IBEX_Spain_ret_20d", "spx_vol_5d", "SJM_JM_Smucker_ret_5d", "JNJ_ret_1d"], "is_new": true}, {"model_id": "new_h10_NORMAL_RandomForest_N15_t1", "algo": "RandomForest", "regime": "NORMAL", "horizon": 10, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWQ_France_zscore_60d", "US1Y_Rate_ret_20d", "PLD_Prologis_ret_5d", "AMGN_Amgen_ret_1d", "EXC_Exelon_zscore_60d", "heston_ev_h3", "EQR_Equity_ret_1d", "MRK_Merck_zscore_60d", "DAX_Germany_vol_20d", "IWM_SmallCap_vol_20d", "TM_Telephone_ret_1d", "SBUX_zscore_60d", "ASX_Australia_vol_20d"], "is_new": true}, {"model_id": "new_h10_NORMAL_RandomForest_N15_t2", "algo": "RandomForest", "regime": "NORMAL", "horizon": 10, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EFFR_ret_1d", "EWJ_Japan_vol_20d", "IWM_SmallCap_vol_20d", "SLB_Schlumberger_ret_1d", "US3M_Rate_zscore_60d", "DAX_Germany_vol_20d", "HD_zscore_60d", "AXP_Amex_ret_20d", "XLY_Disc_vol_20d", "EMR_Emerson_ret_20d", "XOM_ret_20d", "NEE_NextEra_ret_20d", "LLY_zscore_60d"], "is_new": true}, {"model_id": "new_h10_NORMAL_RandomForest_N15_t3", "algo": "RandomForest", "regime": "NORMAL", "horizon": 10, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "IBEX_Spain_ret_20d", "LOW_Lowes_ret_5d", "EWL_Switzerland_vol_20d", "CMCSA_ret_1d", "JNJ_ret_1d", "PG_ret_20d", "heston_var_ev_h5", "AORD_AUS_zscore_60d", "EWA_Australia_ret_1d", "EWM_Malaysia_zscore_60d", "US6M_Rate_ret_20d", "AMGN_Amgen_ret_1d", "BA_ret_1d"], "is_new": true}, {"model_id": "new_h10_NORMAL_RandomForest_N15_t4", "algo": "RandomForest", "regime": "NORMAL", "horizon": 10, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "Nikkei_Japan_zscore_60d", "Retail_Sales_zscore_60d", "US1Y_Rate_ret_20d", "XOM_ret_1d", "SCHW_Schwab_ret_5d", "LMT_LockheedMartin_vol_20d", "HD_zscore_60d", "MS_MorganStanley_zscore_60d", "HangSeng_HK_ret_1d", "AMD_ret_1d", "EWL_Switzerland_vol_20d", "spx_momentum_3d", "AMT_AmericanTower_ret_1d"], "is_new": true}, {"model_id": "new_h10_NORMAL_RandomForest_N15_t5", "algo": "RandomForest", "regime": "NORMAL", "horizon": 10, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "NFCI_ret_5d", "MSTR_Bitcoin3_ret_20d", "EWL_Switzerland_vol_20d", "US6M_Rate_ret_20d", "PAYX_Paychex_zscore_60d", "JNJ_ret_1d", "XLV_Health_zscore_60d", "US7Y_Rate_ret_20d", "TM_Telephone_ret_1d", "HangSeng_HK_vol_20d", "EWS_Singapore_ret_5d", "EQR_Equity_ret_1d", "heston_var_ev_h5"], "is_new": true}, {"model_id": "new_h10_NORMAL_RandomForest_N15_t6", "algo": "RandomForest", "regime": "NORMAL", "horizon": 10, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "VVIX_ret_20d", "AVB_AvalonBay_zscore_60d", "TED_Spread_zscore_60d", "HD_zscore_60d", "AMD_ret_1d", "INTC_ret_1d", "TM_Telephone_ret_1d", "hmm_p_stress", "SBUX_ret_5d", "PG_ret_20d", "EWH_HongKong_ret_5d", "PCAR_PaccarInc_ret_5d", "IBEX_Spain_ret_20d"], "is_new": true}, {"model_id": "new_h10_NORMAL_RandomForest_N15_t7", "algo": "RandomForest", "regime": "NORMAL", "horizon": 10, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "ASX_Australia_ret_5d", "spx_momentum_3d", "DIS_vol_20d", "CLX_Clorox_vol_20d", "3M_ret_5d", "US7Y_Rate_ret_20d", "Core_PCE_zscore_60d", "HD_ret_1d", "XLB_Materials_zscore_60d", "Brent_Oil_FRED_ret_20d", "Michigan_Sentiment_ret_20d", "heston_var_ev_h7", "TM_Telephone_vol_20d"], "is_new": true}, {"model_id": "new_h10_NORMAL_RandomForest_N20_t0", "algo": "RandomForest", "regime": "NORMAL", "horizon": 10, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "US7Y_Rate_ret_20d", "ENB_EnbridgeInc_ret_1d", "DOW_Price_zscore_60d", "DE_Deere_ret_5d", "AVB_AvalonBay_zscore_60d", "3M_vol_20d", "heston_var_ev_h5", "TXN_vol_20d", "FedFunds_zscore_60d", "EXC_Exelon_zscore_60d", "CPB_CampbellSoup_ret_20d", "MSTR_Bitcoin3_ret_20d", "US30Y_Rate_ret_20d", "Nikkei_Japan_vol_20d", "GE_ret_1d", "MRK_Merck_zscore_60d", "PG_ret_20d", "spx_abs_ret_max_5d"], "is_new": true}, {"model_id": "new_h10_NORMAL_RandomForest_N20_t1", "algo": "RandomForest", "regime": "NORMAL", "horizon": 10, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "US3M_Rate_zscore_60d", "BTI_BritishAmerican_ret_20d", "ITT_ITTInc_ret_5d", "LOW_Lowes_ret_5d", "HD_ret_1d", "SLB_Schlumberger_ret_5d", "Retail_Sales_zscore_60d", "DE_Deere_vol_20d", "TED_Spread_zscore_60d", "XLF_Fin_vol_20d", "EQIX_Equinix_ret_5d", "HUM_Humana_ret_5d", "vix_acceleration_1d", "Nikkei_Japan_vol_20d", "SBUX_ret_5d", "EWM_Malaysia_ret_1d", "PAYX_Paychex_zscore_60d", "LMT_LockheedMartin_vol_20d"], "is_new": true}, {"model_id": "new_h10_NORMAL_RandomForest_N20_t2", "algo": "RandomForest", "regime": "NORMAL", "horizon": 10, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "SCHW_Schwab_ret_5d", "IYM_BasicMaterials_ret_20d", "AMZN_ret_5d", "SJM_JM_Smucker_ret_5d", "SPY_zscore_60d", "DE_Deere_vol_20d", "GD_GeneralDynamics_zscore_60d", "MSTR_Bitcoin3_ret_1d", "ES_Evergy_ret_1d", "CTAS_Cintas_vol_20d", "ORCL_vol_20d", "IYR_US_REIT2_zscore_60d", "gjr_condvar_h1", "PPL_PPL_ret_1d", "ASX_Australia_ret_5d", "BA_ret_1d", "TM_Telephone_ret_1d", "EWH_HongKong_ret_5d"], "is_new": true}, {"model_id": "new_h10_NORMAL_RandomForest_N20_t3", "algo": "RandomForest", "regime": "NORMAL", "horizon": 10, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "QQQ_vol_20d", "EOG_EOGResources_ret_5d", "ES_Evergy_ret_1d", "EWQ_France_zscore_60d", "BTI_BritishAmerican_ret_5d", "PPL_PPL_ret_1d", "IYR_US_REIT2_zscore_60d", "JNJ_ret_1d", "heston_var_ev_h5", "CTAS_Cintas_vol_20d", "EWM_Malaysia_ret_1d", "HD_ret_20d", "US1Y_Rate_ret_20d", "SCHW_Schwab_ret_5d", "M_Macys_vol_20d", "PAYX_Paychex_ret_20d", "spx_vol_5d", "ORCL_vol_20d"], "is_new": true}, {"model_id": "new_h10_NORMAL_RandomForest_N20_t4", "algo": "RandomForest", "regime": "NORMAL", "horizon": 10, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "AMD_ret_1d", "Retail_Sales_zscore_60d", "BA_ret_1d", "WTI_Oil_FRED_zscore_60d", "Nikkei_Japan_vol_20d", "EWL_Switzerland_vol_20d", "VRP_ma5", "PAYX_Paychex_zscore_60d", "NOC_Northrop_ret_20d", "HD_ret_1d", "SBUX_zscore_60d", "SJM_JM_Smucker_ret_5d", "DE_Deere_vol_20d", "US3M_Rate_zscore_60d", "EWQ_France_zscore_60d", "MS_MorganStanley_ret_5d", "M_Macys_vol_20d", "PG_ret_20d"], "is_new": true}, {"model_id": "new_h10_NORMAL_RandomForest_N20_t5", "algo": "RandomForest", "regime": "NORMAL", "horizon": 10, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "XOM_ret_1d", "MO_AltriaMG_ret_1d", "AXP_Amex_ret_20d", "DHR_vol_20d", "NOC_Northrop_ret_20d", "HD_ret_1d", "Brent_Oil_FRED_ret_5d", "TM_Telephone_vol_20d", "PAYX_Paychex_ret_20d", "ES_Evergy_ret_1d", "EWL_Switzerland_zscore_60d", "TGT_Target_zscore_60d", "VVIX_ret_20d", "MSTR_Bitcoin3_ret_20d", "US3Y_Rate_ret_5d", "Nikkei_Japan_vol_20d", "vix_mean_abs_ret_5d", "TED_Spread_vol_20d"], "is_new": true}, {"model_id": "new_h10_NORMAL_RandomForest_N20_t6", "algo": "RandomForest", "regime": "NORMAL", "horizon": 10, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "AORD_AUS_zscore_60d", "NFCI_ret_5d", "IBEX_Spain_ret_20d", "EOG_EOGResources_ret_5d", "EWA_Australia_zscore_60d", "EFFR_vol_20d", "Core_PCE_zscore_60d", "US1Y_Rate_ret_5d", "BA_ret_1d", "US1Y_Rate_ret_20d", "CTAS_Cintas_vol_20d", "heston_var_ev_h7", "Core_CPI_zscore_60d", "M_Macys_vol_20d", "NWL_Newell_ret_20d", "TM_Telephone_ret_1d", "XLV_Health_zscore_60d", "ORCL_vol_20d"], "is_new": true}, {"model_id": "new_h10_NORMAL_RandomForest_N20_t7", "algo": "RandomForest", "regime": "NORMAL", "horizon": 10, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWQ_France_zscore_60d", "XLV_Health_zscore_60d", "EWL_Switzerland_vol_20d", "DOW_Price_zscore_60d", "EWA_Australia_zscore_60d", "HangSeng_HK_ret_5d", "US3Y_Rate_ret_5d", "PAYX_Paychex_zscore_60d", "ENB_EnbridgeInc_ret_1d", "AMD_ret_5d", "SLB_Schlumberger_ret_5d", "EWY_Korea_ret_20d", "EWY_Korea_zscore_60d", "SBUX_vol_20d", "AVB_AvalonBay_zscore_60d", "TED_Spread_zscore_60d", "AMD_ret_1d", "HD_ret_20d"], "is_new": true}, {"model_id": "new_h10_NORMAL_RandomForest_N25_t0", "algo": "RandomForest", "regime": "NORMAL", "horizon": 10, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "ASX_Australia_ret_5d", "EWY_Korea_ret_20d", "EXC_Exelon_zscore_60d", "WTI_Oil_FRED_zscore_60d", "vix_acceleration_1d", "HD_zscore_60d", "EQR_Equity_ret_1d", "spx_abs_ret_max_5d", "EWJ_Japan_vol_20d", "AXP_Amex_ret_20d", "ITT_ITTInc_ret_5d", "MSTR_Bitcoin3_ret_5d", "SJM_JM_Smucker_ret_5d", "DHR_vol_20d", "DAX_Germany_vol_20d", "XLV_Health_zscore_60d", "BDX_Becton_Dickinson_ret_20d", "heston_var_ev_h7", "EWG_Germany_vol_20d", "BTI_BritishAmerican_ret_20d", "ASX_Australia_vol_20d", "EWQ_France_ret_20d", "TXN_vol_20d"], "is_new": true}, {"model_id": "new_h10_NORMAL_RandomForest_N25_t1", "algo": "RandomForest", "regime": "NORMAL", "horizon": 10, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "NWL_Newell_ret_20d", "HD_ret_20d", "US7Y_Rate_ret_20d", "DAX_Germany_vol_20d", "EWM_Malaysia_vol_20d", "EWA_Australia_zscore_60d", "MSTR_Bitcoin3_ret_20d", "HangSeng_HK_ret_5d", "AXP_Amex_vol_20d", "DE_Deere_ret_5d", "EMR_Emerson_ret_20d", "BLK_BlackRock_zscore_60d", "SLB_Schlumberger_ret_1d", "EFFR_vol_20d", "JNJ_ret_1d", "SBUX_vol_20d", "CLX_Clorox_vol_20d", "AXP_Amex_ret_20d", "heston_var_ev_h3", "gjr_condvar_h1", "NVDA_vol_20d", "EOG_EOGResources_vol_20d", "EWQ_France_ret_20d"], "is_new": true}, {"model_id": "new_h10_NORMAL_RandomForest_N25_t2", "algo": "RandomForest", "regime": "NORMAL", "horizon": 10, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "IWM_SmallCap_vol_20d", "heston_var_ev_h3", "EWQ_France_ret_20d", "Michigan_Sentiment_ret_20d", "DHR_ret_1d", "HangSeng_HK_vol_20d", "TED_Spread_zscore_60d", "ORCL_zscore_60d", "SJM_JM_Smucker_ret_5d", "GD_GeneralDynamics_zscore_60d", "SLB_Schlumberger_ret_5d", "LUV_SouthwestAir_ret_5d", "MS_MorganStanley_zscore_60d", "DAX_Germany_vol_20d", "EFFR_vol_20d", "US30Y_Rate_ret_20d", "PAYX_Paychex_ret_20d", "HUM_Humana_ret_5d", "US7Y_Rate_ret_20d", "CLX_Clorox_vol_20d", "PPL_PPL_ret_1d", "MS_MorganStanley_ret_5d", "AXP_Amex_ret_20d"], "is_new": true}, {"model_id": "new_h10_NORMAL_RandomForest_N25_t3", "algo": "RandomForest", "regime": "NORMAL", "horizon": 10, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "LMT_LockheedMartin_vol_20d", "EXC_Exelon_zscore_60d", "EOG_EOGResources_vol_20d", "AMZN_ret_5d", "CPB_CampbellSoup_ret_5d", "Core_CPI_zscore_60d", "QQQ_vol_20d", "EWG_Germany_vol_20d", "TXN_vol_20d", "VVIX_ret_20d", "XLF_Fin_vol_20d", "DAX_Germany_zscore_60d", "CMCSA_ret_1d", "HD_ret_1d", "NOC_Northrop_ret_20d", "DAX_Germany_vol_20d", "CLX_Clorox_vol_20d", "US7Y_Rate_ret_20d", "XLK_Tech_zscore_60d", "US5Y_Rate_ret_5d", "SLB_Schlumberger_ret_5d", "EWG_Germany_ret_20d", "US3Y_Rate_ret_5d"], "is_new": true}, {"model_id": "new_h10_NORMAL_RandomForest_N25_t4", "algo": "RandomForest", "regime": "NORMAL", "horizon": 10, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "spx_momentum_3d", "EWM_Malaysia_vol_20d", "Core_PCE_zscore_60d", "IWM_SmallCap_vol_20d", "EWJ_Japan_vol_20d", "ASX_Australia_vol_20d", "Nikkei_Japan_vol_20d", "CPB_CampbellSoup_ret_5d", "3M_ret_5d", "TED_Spread_zscore_60d", "GD_GeneralDynamics_zscore_60d", "XOM_ret_20d", "EWG_Germany_ret_20d", "EFFR_vol_20d", "EWS_Singapore_ret_5d", "CTAS_Cintas_vol_20d", "XLY_Disc_vol_20d", "BA_ret_1d", "CCI_CrownCastle_vol_20d", "HangSeng_HK_ret_5d", "heston_ev_h3", "PLD_Prologis_ret_5d", "vix_acceleration_1d"], "is_new": true}, {"model_id": "new_h10_NORMAL_RandomForest_N25_t5", "algo": "RandomForest", "regime": "NORMAL", "horizon": 10, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "DHR_vol_20d", "MRK_Merck_zscore_60d", "XLV_Health_zscore_60d", "PAYX_Paychex_zscore_60d", "HD_ret_1d", "AMGN_Amgen_ret_1d", "Core_PCE_zscore_60d", "INTC_ret_1d", "CPB_CampbellSoup_vol_20d", "SBUX_zscore_60d", "DAX_Germany_vol_20d", "HUM_Humana_ret_5d", "EWJ_Japan_vol_20d", "LOW_Lowes_ret_20d", "MS_MorganStanley_zscore_60d", "DOW_Price_zscore_60d", "SJM_JM_Smucker_ret_5d", "EMR_Emerson_ret_20d", "LLY_zscore_60d", "BTI_BritishAmerican_ret_5d", "LOW_Lowes_ret_5d", "GILD_Gilead_ret_20d", "hmm_p_stress"], "is_new": true}, {"model_id": "new_h10_NORMAL_RandomForest_N25_t6", "algo": "RandomForest", "regime": "NORMAL", "horizon": 10, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWL_Switzerland_vol_20d", "AXP_Amex_ret_20d", "Core_PCE_zscore_60d", "HD_ret_20d", "gjr_condvar_h1", "heston_var_ev_h3", "LMT_LockheedMartin_ret_1d", "CPB_CampbellSoup_vol_20d", "Michigan_Sentiment_ret_20d", "FedFunds_zscore_60d", "XLK_Tech_zscore_60d", "TM_Telephone_ret_1d", "SBUX_vol_20d", "NOC_Northrop_ret_20d", "PCAR_PaccarInc_ret_5d", "GD_GeneralDynamics_zscore_60d", "Nikkei_Japan_zscore_60d", "SLB_Schlumberger_ret_5d", "GILD_Gilead_ret_20d", "XLV_Health_zscore_60d", "AXP_Amex_vol_20d", "hmm_p_stress", "DE_Deere_ret_5d"], "is_new": true}, {"model_id": "new_h10_NORMAL_RandomForest_N25_t7", "algo": "RandomForest", "regime": "NORMAL", "horizon": 10, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "WTI_Oil_FRED_zscore_60d", "EWL_Switzerland_vol_20d", "AORD_AUS_zscore_60d", "XOM_ret_1d", "Brent_Oil_FRED_ret_5d", "SLB_Schlumberger_ret_5d", "HD_ret_20d", "TGT_Target_zscore_60d", "SO_SouthernCo_ret_5d", "EFFR_vol_20d", "EXC_Exelon_ret_1d", "DOW_Price_zscore_60d", "BTI_BritishAmerican_ret_20d", "Brent_Oil_FRED_ret_20d", "AMT_AmericanTower_ret_1d", "SCHW_Schwab_ret_5d", "ASX_Australia_ret_5d", "DE_Deere_vol_20d", "TM_Telephone_vol_20d", "T10Y2Y_Spread_ret_5d", "EWQ_France_ret_20d", "NOC_Northrop_ret_20d", "DAX_Germany_vol_20d"], "is_new": true}, {"model_id": "new_h10_NORMAL_RandomForest_N30_t0", "algo": "RandomForest", "regime": "NORMAL", "horizon": 10, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWM_Malaysia_ret_1d", "MSTR_Bitcoin3_ret_1d", "GD_GeneralDynamics_zscore_60d", "IWM_SmallCap_vol_20d", "gjr_condvar_h1", "FedFunds_zscore_60d", "XLY_Disc_vol_20d", "INTC_ret_1d", "AMGN_Amgen_ret_1d", "US3M_Rate_vol_20d", "DE_Deere_vol_20d", "HD_zscore_60d", "EWG_Germany_vol_20d", "CPB_CampbellSoup_ret_5d", "HangSeng_HK_vol_20d", "TM_Telephone_ret_1d", "CCI_CrownCastle_vol_20d", "T_ret_1d", "ORCL_vol_20d", "EXC_Exelon_zscore_60d", "IYR_US_REIT2_zscore_60d", "CLX_Clorox_vol_20d", "MS_MorganStanley_zscore_60d", "ES_Evergy_ret_1d", "MRK_Merck_zscore_60d", "BTI_BritishAmerican_ret_20d", "AORD_AUS_zscore_60d", "hmm_p_stress"], "is_new": true}, {"model_id": "new_h10_NORMAL_RandomForest_N30_t1", "algo": "RandomForest", "regime": "NORMAL", "horizon": 10, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "QQQ_vol_20d", "IBEX_Spain_ret_20d", "CPB_CampbellSoup_ret_20d", "HangSeng_HK_ret_5d", "HD_ret_20d", "3M_ret_5d", "ENB_EnbridgeInc_ret_1d", "heston_var_ev_h7", "Core_CPI_zscore_60d", "CPB_CampbellSoup_vol_20d", "BLK_BlackRock_zscore_60d", "SLB_Schlumberger_ret_5d", "CPB_CampbellSoup_zscore_60d", "Core_PCE_zscore_60d", "XLY_Disc_vol_20d", "US7Y_Rate_ret_20d", "DE_Deere_vol_20d", "heston_var_ev_h3", "LOW_Lowes_ret_5d", "DOW_Price_zscore_60d", "ASX_Australia_ret_5d", "hmm_p_stress", "AMD_ret_5d", "XLB_Materials_zscore_60d", "US3M_Rate_vol_20d", "Nikkei_Japan_zscore_60d", "LLY_zscore_60d", "NWL_Newell_ret_20d"], "is_new": true}, {"model_id": "new_h10_NORMAL_RandomForest_N30_t2", "algo": "RandomForest", "regime": "NORMAL", "horizon": 10, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "TED_Spread_zscore_60d", "EWC_Canada_zscore_60d", "HangSeng_HK_vol_20d", "GILD_Gilead_ret_20d", "PPL_PPL_ret_1d", "SO_SouthernCo_ret_5d", "WTI_Oil_FRED_zscore_60d", "spx_abs_ret_max_5d", "DAX_Germany_vol_20d", "EQR_Equity_ret_1d", "Retail_Sales_zscore_60d", "ORCL_zscore_60d", "MS_MorganStanley_zscore_60d", "PLD_Prologis_ret_5d", "heston_ev_h3", "gjr_condvar_h1", "BTI_BritishAmerican_ret_20d", "ASX_Australia_ret_5d", "NWL_Newell_ret_20d", "PAYX_Paychex_zscore_60d", "hmm_p_stress", "SJM_JM_Smucker_ret_5d", "JNJ_ret_1d", "vix_acceleration_1d", "PAYX_Paychex_ret_20d", "NFCI_ret_5d", "TXN_vol_20d", "EWA_Australia_ret_1d"], "is_new": true}, {"model_id": "new_h10_NORMAL_RandomForest_N30_t3", "algo": "RandomForest", "regime": "NORMAL", "horizon": 10, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "Michigan_Sentiment_ret_20d", "EFFR_ret_1d", "EQR_Equity_ret_1d", "AXP_Amex_ret_20d", "QQQ_vol_20d", "heston_var_ev_h5", "NEE_NextEra_ret_20d", "VRP_ma5", "EWG_Germany_vol_20d", "PPL_PPL_ret_1d", "PG_ret_20d", "EXC_Exelon_zscore_60d", "INTC_ret_5d", "HUM_Humana_ret_5d", "DIS_vol_20d", "AMZN_ret_5d", "hmm_p_stress", "US5Y_Rate_ret_5d", "spx_abs_ret_max_5d", "EQIX_Equinix_ret_5d", "EWL_Switzerland_zscore_60d", "NOC_Northrop_ret_20d", "INTC_ret_1d", "SJM_JM_Smucker_ret_5d", "PAYX_Paychex_zscore_60d", "TM_Telephone_vol_20d", "GILD_Gilead_ret_20d", "M_Macys_vol_20d"], "is_new": true}, {"model_id": "new_h10_NORMAL_RandomForest_N30_t4", "algo": "RandomForest", "regime": "NORMAL", "horizon": 10, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EXC_Exelon_zscore_60d", "ASX_Australia_ret_5d", "TM_Telephone_vol_20d", "gjr_condvar_h1", "EFFR_vol_20d", "DIS_vol_20d", "HangSeng_HK_ret_5d", "CPB_CampbellSoup_ret_20d", "AMZN_ret_5d", "US5Y_Rate_ret_5d", "AMGN_Amgen_ret_1d", "DHR_ret_1d", "CI_Cigna_vol_20d", "Brent_Oil_FRED_ret_5d", "EWM_Malaysia_zscore_60d", "ORCL_vol_20d", "ASX_Australia_vol_20d", "BDX_Becton_Dickinson_ret_20d", "Industrial_Production_zscore_60d", "PAYX_Paychex_zscore_60d", "SLB_Schlumberger_ret_1d", "SPY_zscore_60d", "EWQ_France_ret_20d", "US1Y_Rate_ret_5d", "MRK_Merck_zscore_60d", "XLF_Fin_vol_20d", "EFFR_ret_1d", "ORCL_zscore_60d"], "is_new": true}, {"model_id": "new_h10_NORMAL_RandomForest_N30_t5", "algo": "RandomForest", "regime": "NORMAL", "horizon": 10, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "TM_Telephone_vol_20d", "SBUX_vol_20d", "EFFR_ret_1d", "BDX_Becton_Dickinson_ret_20d", "IBEX_Spain_ret_20d", "SBUX_zscore_60d", "MS_MorganStanley_ret_1d", "heston_ev_h3", "LOW_Lowes_ret_5d", "US3Y_Rate_ret_5d", "AMGN_Amgen_ret_1d", "US1Y_Rate_ret_5d", "US30Y_Rate_ret_20d", "US1Y_Rate_ret_20d", "EQIX_Equinix_ret_5d", "LMT_LockheedMartin_ret_1d", "spx_momentum_3d", "Nikkei_Japan_zscore_60d", "Michigan_Sentiment_ret_20d", "AXP_Amex_ret_20d", "Nikkei_Japan_vol_20d", "HD_ret_20d", "WTI_Oil_FRED_zscore_60d", "SJM_JM_Smucker_ret_1d", "vix_mean_abs_ret_5d", "DOW_Price_zscore_60d", "M_Macys_vol_20d", "SLB_Schlumberger_ret_1d"], "is_new": true}, {"model_id": "new_h10_NORMAL_RandomForest_N30_t6", "algo": "RandomForest", "regime": "NORMAL", "horizon": 10, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "CPB_CampbellSoup_ret_5d", "PAYX_Paychex_ret_20d", "EXC_Exelon_zscore_60d", "IYM_BasicMaterials_ret_20d", "LUV_SouthwestAir_ret_5d", "EWJ_Japan_vol_20d", "US3M_Rate_vol_20d", "EWG_Germany_vol_20d", "EWY_Korea_zscore_60d", "spx_abs_ret_max_5d", "SJM_JM_Smucker_ret_5d", "US5Y_Rate_ret_5d", "CMCSA_ret_1d", "HD_ret_20d", "EWM_Malaysia_vol_20d", "EWL_Switzerland_vol_20d", "PLD_Prologis_ret_5d", "GILD_Gilead_ret_20d", "BTI_BritishAmerican_ret_5d", "MO_AltriaMG_ret_1d", "PAYX_Paychex_vol_20d", "DOW_Price_zscore_60d", "TED_Spread_zscore_60d", "CPB_CampbellSoup_zscore_60d", "EOG_EOGResources_vol_20d", "DHR_vol_20d", "NEE_NextEra_ret_20d", "FedFunds_zscore_60d"], "is_new": true}, {"model_id": "new_h10_NORMAL_RandomForest_N30_t7", "algo": "RandomForest", "regime": "NORMAL", "horizon": 10, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "MSTR_Bitcoin3_ret_1d", "EWA_Australia_ret_1d", "DIS_vol_20d", "XLK_Tech_zscore_60d", "EXC_Exelon_zscore_60d", "spx_vol_5d", "GILD_Gilead_ret_20d", "SBUX_zscore_60d", "HangSeng_HK_ret_5d", "XLV_Health_zscore_60d", "EWQ_France_zscore_60d", "TM_Telephone_ret_1d", "US30Y_Rate_ret_20d", "EFFR_ret_1d", "PG_ret_20d", "US5Y_Rate_ret_5d", "Nikkei_Japan_vol_20d", "EWA_Australia_zscore_60d", "BTI_BritishAmerican_ret_5d", "3M_ret_5d", "CTAS_Cintas_vol_20d", "EFFR_vol_20d", "EWM_Malaysia_ret_1d", "HD_ret_5d", "EWJ_Japan_vol_20d", "LUV_SouthwestAir_ret_5d", "HangSeng_HK_vol_20d", "ES_Evergy_ret_1d"], "is_new": true}, {"model_id": "new_h10_NORMAL_LogisticRegression_N5_t0", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 10, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "Michigan_Sentiment_ret_20d", "heston_var_ev_h7", "VOD_Vodafone_zscore_60d"], "is_new": true}, {"model_id": "new_h10_NORMAL_LogisticRegression_N5_t1", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 10, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "TXN_vol_20d", "EWY_Korea_ret_20d", "EWM_Malaysia_ret_1d"], "is_new": true}, {"model_id": "new_h10_NORMAL_LogisticRegression_N5_t2", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 10, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "CPB_CampbellSoup_vol_20d", "BTI_BritishAmerican_ret_20d", "NOC_Northrop_ret_20d"], "is_new": true}, {"model_id": "new_h10_NORMAL_LogisticRegression_N5_t3", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 10, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "US1Y_Rate_ret_5d", "US1Y_Rate_ret_20d", "CI_Cigna_vol_20d"], "is_new": true}, {"model_id": "new_h10_NORMAL_LogisticRegression_N5_t4", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 10, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "SLB_Schlumberger_ret_1d", "HUM_Humana_ret_5d", "PCAR_PaccarInc_ret_5d"], "is_new": true}, {"model_id": "new_h10_NORMAL_LogisticRegression_N5_t5", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 10, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "SPY_zscore_60d", "MO_AltriaMG_ret_1d", "INTC_ret_1d"], "is_new": true}, {"model_id": "new_h10_NORMAL_LogisticRegression_N5_t6", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 10, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "GE_ret_1d", "EWA_Australia_ret_1d", "HD_zscore_60d"], "is_new": true}, {"model_id": "new_h10_NORMAL_LogisticRegression_N5_t7", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 10, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "US1Y_Rate_ret_5d", "EWG_Germany_ret_20d", "XLY_Disc_vol_20d"], "is_new": true}, {"model_id": "new_h10_NORMAL_LogisticRegression_N8_t0", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 10, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "M_Macys_vol_20d", "DOW_Price_zscore_60d", "PAYX_Paychex_vol_20d", "MSTR_Bitcoin3_ret_20d", "ES_Evergy_ret_1d", "EFFR_ret_1d"], "is_new": true}, {"model_id": "new_h10_NORMAL_LogisticRegression_N8_t1", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 10, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "heston_var_ev_h3", "MS_MorganStanley_zscore_60d", "NOC_Northrop_ret_20d", "CLX_Clorox_vol_20d", "SLB_Schlumberger_ret_1d", "vix_acceleration_1d"], "is_new": true}, {"model_id": "new_h10_NORMAL_LogisticRegression_N8_t2", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 10, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "ORCL_zscore_60d", "heston_var_ev_h3", "GILD_Gilead_ret_20d", "CPB_CampbellSoup_zscore_60d", "AMZN_ret_5d", "spx_momentum_3d"], "is_new": true}, {"model_id": "new_h10_NORMAL_LogisticRegression_N8_t3", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 10, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "US3Y_Rate_ret_5d", "DE_Deere_ret_5d", "NEE_NextEra_ret_20d", "AMZN_ret_5d", "SLB_Schlumberger_ret_1d", "EQIX_Equinix_ret_5d"], "is_new": true}, {"model_id": "new_h10_NORMAL_LogisticRegression_N8_t4", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 10, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "AMD_ret_1d", "US3M_Rate_vol_20d", "AMZN_ret_5d", "DAX_Germany_vol_20d", "ASX_Australia_vol_20d", "US1Y_Rate_ret_20d"], "is_new": true}, {"model_id": "new_h10_NORMAL_LogisticRegression_N8_t5", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 10, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "ASX_Australia_ret_5d", "AMT_AmericanTower_ret_1d", "LUV_SouthwestAir_ret_5d", "EOG_EOGResources_vol_20d", "HangSeng_HK_ret_1d", "gjr_condvar_h1"], "is_new": true}, {"model_id": "new_h10_NORMAL_LogisticRegression_N8_t6", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 10, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "AXP_Amex_vol_20d", "XLY_Disc_vol_20d", "3M_ret_5d", "EWH_HongKong_ret_5d", "SJM_JM_Smucker_ret_5d", "EOG_EOGResources_ret_5d"], "is_new": true}, {"model_id": "new_h10_NORMAL_LogisticRegression_N8_t7", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 10, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "vix_mean_abs_ret_5d", "IYR_US_REIT2_zscore_60d", "CPB_CampbellSoup_vol_20d", "BLK_BlackRock_zscore_60d", "Core_PCE_zscore_60d", "EWM_Malaysia_vol_20d"], "is_new": true}, {"model_id": "new_h10_NORMAL_LogisticRegression_N10_t0", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 10, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "QQQ_vol_20d", "XOM_ret_20d", "EWC_Canada_zscore_60d", "DE_Deere_ret_5d", "WTI_Oil_FRED_zscore_60d", "AXP_Amex_ret_20d", "gjr_condvar_h1", "PPL_PPL_ret_1d"], "is_new": true}, {"model_id": "new_h10_NORMAL_LogisticRegression_N10_t1", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 10, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "IYM_BasicMaterials_ret_20d", "heston_ev_h3", "TED_Spread_zscore_60d", "DE_Deere_vol_20d", "EQIX_Equinix_ret_5d", "EWH_HongKong_ret_5d", "EWJ_Japan_vol_20d", "PPL_PPL_ret_1d"], "is_new": true}, {"model_id": "new_h10_NORMAL_LogisticRegression_N10_t2", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 10, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "BLK_BlackRock_zscore_60d", "TM_Telephone_vol_20d", "3M_ret_5d", "SBUX_vol_20d", "CPB_CampbellSoup_vol_20d", "AMD_ret_1d", "Core_PCE_zscore_60d", "LMT_LockheedMartin_vol_20d"], "is_new": true}, {"model_id": "new_h10_NORMAL_LogisticRegression_N10_t3", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 10, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "US6M_Rate_ret_20d", "T_ret_1d", "IWM_SmallCap_vol_20d", "US3M_Rate_vol_20d", "PPL_PPL_ret_1d", "BDX_Becton_Dickinson_ret_20d", "EWJ_Japan_vol_20d", "PLD_Prologis_ret_5d"], "is_new": true}, {"model_id": "new_h10_NORMAL_LogisticRegression_N10_t4", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 10, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "SPY_zscore_60d", "DHR_ret_1d", "XLK_Tech_zscore_60d", "LOW_Lowes_ret_20d", "spx_momentum_3d", "WTI_Oil_FRED_zscore_60d", "HD_ret_20d", "NVDA_vol_20d"], "is_new": true}, {"model_id": "new_h10_NORMAL_LogisticRegression_N10_t5", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 10, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "US7Y_Rate_ret_20d", "EWQ_France_zscore_60d", "EWL_Switzerland_zscore_60d", "PAYX_Paychex_vol_20d", "IYR_US_REIT2_zscore_60d", "EMR_Emerson_ret_20d", "BTI_BritishAmerican_ret_20d", "LOW_Lowes_ret_5d"], "is_new": true}, {"model_id": "new_h10_NORMAL_LogisticRegression_N10_t6", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 10, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "PAYX_Paychex_zscore_60d", "CTAS_Cintas_vol_20d", "EWA_Australia_ret_1d", "MS_MorganStanley_ret_1d", "XLK_Tech_zscore_60d", "heston_var_ev_h5", "vix_mean_abs_ret_5d", "Retail_Sales_zscore_60d"], "is_new": true}, {"model_id": "new_h10_NORMAL_LogisticRegression_N10_t7", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 10, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWJ_Japan_vol_20d", "GE_ret_1d", "CMCSA_ret_1d", "EWL_Switzerland_zscore_60d", "ASX_Australia_ret_5d", "ORCL_vol_20d", "EWL_Switzerland_vol_20d", "Michigan_Sentiment_ret_20d"], "is_new": true}, {"model_id": "new_h10_NORMAL_LogisticRegression_N12_t0", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 10, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "vix_acceleration_1d", "EWM_Malaysia_zscore_60d", "AMD_ret_1d", "EXC_Exelon_ret_1d", "FedFunds_zscore_60d", "MSTR_Bitcoin3_ret_1d", "BTI_BritishAmerican_ret_20d", "MO_AltriaMG_ret_1d", "MRK_Merck_zscore_60d", "XOM_ret_20d"], "is_new": true}, {"model_id": "new_h10_NORMAL_LogisticRegression_N12_t1", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 10, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "ENB_EnbridgeInc_ret_1d", "PFE_ret_1d", "SJM_JM_Smucker_ret_1d", "AVB_AvalonBay_zscore_60d", "EFFR_vol_20d", "IWM_SmallCap_vol_20d", "EWY_Korea_ret_20d", "SJM_JM_Smucker_ret_5d", "CPB_CampbellSoup_ret_20d", "LMT_LockheedMartin_ret_1d"], "is_new": true}, {"model_id": "new_h10_NORMAL_LogisticRegression_N12_t2", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 10, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "SPY_zscore_60d", "GILD_Gilead_ret_20d", "QQQ_vol_20d", "HD_zscore_60d", "MSTR_Bitcoin3_ret_1d", "CCI_CrownCastle_vol_20d", "EOG_EOGResources_vol_20d", "ORCL_zscore_60d", "IYR_US_REIT2_zscore_60d", "3M_vol_20d"], "is_new": true}, {"model_id": "new_h10_NORMAL_LogisticRegression_N12_t3", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 10, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "BDX_Becton_Dickinson_ret_20d", "XLF_Fin_vol_20d", "EWA_Australia_ret_1d", "MS_MorganStanley_ret_1d", "Core_CPI_zscore_60d", "EOG_EOGResources_ret_5d", "AVB_AvalonBay_zscore_60d", "Brent_Oil_FRED_ret_5d", "SBUX_vol_20d", "NWL_Newell_ret_20d"], "is_new": true}, {"model_id": "new_h10_NORMAL_LogisticRegression_N12_t4", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 10, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "TED_Spread_zscore_60d", "heston_ev_h3", "AXP_Amex_vol_20d", "PFE_ret_1d", "PCAR_PaccarInc_ret_5d", "EMR_Emerson_ret_20d", "CMCSA_ret_1d", "EFFR_ret_1d", "CI_Cigna_vol_20d", "DHR_vol_20d"], "is_new": true}, {"model_id": "new_h10_NORMAL_LogisticRegression_N12_t5", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 10, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "DOW_Price_zscore_60d", "JNJ_ret_1d", "CCI_CrownCastle_vol_20d", "EOG_EOGResources_vol_20d", "IBEX_Spain_ret_20d", "DIS_vol_20d", "Nikkei_Japan_vol_20d", "PAYX_Paychex_ret_20d", "EWS_Singapore_ret_5d", "US3M_Rate_zscore_60d"], "is_new": true}, {"model_id": "new_h10_NORMAL_LogisticRegression_N12_t6", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 10, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "heston_var_ev_h7", "Core_PCE_zscore_60d", "EQIX_Equinix_ret_5d", "DE_Deere_vol_20d", "HD_ret_1d", "MS_MorganStanley_zscore_60d", "CPB_CampbellSoup_ret_20d", "PPL_PPL_ret_1d", "HD_ret_5d", "Michigan_Sentiment_ret_20d"], "is_new": true}, {"model_id": "new_h10_NORMAL_LogisticRegression_N12_t7", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 10, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "NVDA_vol_20d", "AMD_ret_5d", "EWG_Germany_vol_20d", "IYM_BasicMaterials_ret_20d", "Nikkei_Japan_vol_20d", "US7Y_Rate_ret_20d", "NOC_Northrop_ret_20d", "Michigan_Sentiment_ret_20d", "CI_Cigna_vol_20d", "Industrial_Production_zscore_60d"], "is_new": true}, {"model_id": "new_h10_NORMAL_LogisticRegression_N15_t0", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 10, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "SPY_zscore_60d", "TGT_Target_zscore_60d", "Industrial_Production_zscore_60d", "DAX_Germany_vol_20d", "HangSeng_HK_vol_20d", "CPB_CampbellSoup_ret_20d", "ENB_EnbridgeInc_ret_1d", "XLF_Fin_vol_20d", "EFFR_ret_1d", "PG_ret_20d", "LUV_SouthwestAir_ret_5d", "MSTR_Bitcoin3_ret_20d", "MRK_Merck_zscore_60d"], "is_new": true}, {"model_id": "new_h10_NORMAL_LogisticRegression_N15_t1", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 10, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "TM_Telephone_ret_1d", "PFE_ret_1d", "vix_acceleration_1d", "VRP_ma5", "AXP_Amex_vol_20d", "CCI_CrownCastle_vol_20d", "EWG_Germany_vol_20d", "EWC_Canada_zscore_60d", "gjr_condvar_h1", "LMT_LockheedMartin_ret_1d", "heston_var_ev_h3", "BDX_Becton_Dickinson_ret_20d", "M_Macys_vol_20d"], "is_new": true}, {"model_id": "new_h10_NORMAL_LogisticRegression_N15_t2", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 10, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWA_Australia_zscore_60d", "LMT_LockheedMartin_ret_1d", "XLV_Health_zscore_60d", "Brent_Oil_FRED_ret_20d", "EWG_Germany_ret_20d", "LOW_Lowes_ret_5d", "Michigan_Sentiment_ret_20d", "PAYX_Paychex_ret_20d", "US1Y_Rate_ret_20d", "QQQ_vol_20d", "SBUX_vol_20d", "HangSeng_HK_ret_5d", "DAX_Germany_zscore_60d"], "is_new": true}, {"model_id": "new_h10_NORMAL_LogisticRegression_N15_t3", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 10, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "Retail_Sales_zscore_60d", "BTI_BritishAmerican_ret_20d", "INTC_ret_5d", "QQQ_vol_20d", "GE_ret_1d", "DAX_Germany_zscore_60d", "NVDA_vol_20d", "WTI_Oil_FRED_zscore_60d", "MS_MorganStanley_ret_1d", "XLB_Materials_zscore_60d", "EWC_Canada_zscore_60d", "EXC_Exelon_zscore_60d", "HangSeng_HK_ret_1d"], "is_new": true}, {"model_id": "new_h10_NORMAL_LogisticRegression_N15_t4", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 10, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "AORD_AUS_zscore_60d", "INTC_ret_1d", "CCI_CrownCastle_vol_20d", "EWA_Australia_zscore_60d", "TXN_vol_20d", "CI_Cigna_vol_20d", "CPB_CampbellSoup_vol_20d", "IYR_US_REIT2_zscore_60d", "LLY_zscore_60d", "CTAS_Cintas_vol_20d", "MSTR_Bitcoin3_ret_5d", "EWQ_France_ret_20d", "IBEX_Spain_ret_20d"], "is_new": true}, {"model_id": "new_h10_NORMAL_LogisticRegression_N15_t5", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 10, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "Core_PCE_zscore_60d", "DAX_Germany_zscore_60d", "EWG_Germany_ret_20d", "HD_ret_5d", "ENB_EnbridgeInc_ret_1d", "EQR_Equity_ret_1d", "US5Y_Rate_ret_5d", "NVDA_vol_20d", "GILD_Gilead_ret_20d", "EMR_Emerson_ret_20d", "TM_Telephone_vol_20d", "WTI_Oil_FRED_zscore_60d", "IYR_US_REIT2_zscore_60d"], "is_new": true}, {"model_id": "new_h10_NORMAL_LogisticRegression_N15_t6", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 10, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "3M_vol_20d", "PAYX_Paychex_zscore_60d", "EXC_Exelon_zscore_60d", "CLX_Clorox_vol_20d", "LOW_Lowes_ret_5d", "PG_ret_20d", "EQIX_Equinix_ret_5d", "EWM_Malaysia_ret_1d", "T10Y2Y_Spread_ret_5d", "spx_abs_ret_max_5d", "VRP_ma5", "Industrial_Production_zscore_60d", "HD_zscore_60d"], "is_new": true}, {"model_id": "new_h10_NORMAL_LogisticRegression_N15_t7", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 10, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "US3M_Rate_zscore_60d", "EWY_Korea_ret_20d", "AMGN_Amgen_ret_1d", "HD_ret_5d", "DOW_Price_zscore_60d", "heston_var_ev_h5", "ASX_Australia_ret_5d", "SJM_JM_Smucker_ret_1d", "CCI_CrownCastle_vol_20d", "AMD_ret_5d", "DAX_Germany_vol_20d", "GE_ret_1d", "EFFR_ret_1d"], "is_new": true}, {"model_id": "new_h10_NORMAL_LogisticRegression_N20_t0", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 10, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWL_Switzerland_zscore_60d", "XOM_ret_1d", "ASX_Australia_ret_5d", "EWM_Malaysia_zscore_60d", "VVIX_ret_20d", "Brent_Oil_FRED_ret_5d", "IWM_SmallCap_vol_20d", "WTI_Oil_FRED_zscore_60d", "IYM_BasicMaterials_ret_20d", "XLY_Disc_vol_20d", "Michigan_Sentiment_ret_20d", "BA_ret_1d", "heston_var_ev_h3", "EWQ_France_zscore_60d", "TM_Telephone_ret_1d", "HD_ret_20d", "HD_ret_5d", "LMT_LockheedMartin_ret_1d"], "is_new": true}, {"model_id": "new_h10_NORMAL_LogisticRegression_N20_t1", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 10, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "TED_Spread_zscore_60d", "spx_vol_5d", "XLY_Disc_vol_20d", "gjr_condvar_h1", "EWS_Singapore_ret_5d", "LMT_LockheedMartin_vol_20d", "Brent_Oil_FRED_ret_5d", "Nikkei_Japan_zscore_60d", "PG_ret_20d", "JNJ_ret_1d", "AMGN_Amgen_ret_1d", "AXP_Amex_vol_20d", "HD_ret_5d", "NEE_NextEra_ret_20d", "CI_Cigna_vol_20d", "MSTR_Bitcoin3_ret_20d", "MSTR_Bitcoin3_ret_1d", "HD_ret_1d"], "is_new": true}, {"model_id": "new_h10_NORMAL_LogisticRegression_N20_t2", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 10, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWL_Switzerland_vol_20d", "DE_Deere_vol_20d", "vix_mean_abs_ret_5d", "HUM_Humana_ret_5d", "XLB_Materials_zscore_60d", "EQR_Equity_ret_1d", "EWY_Korea_ret_20d", "TM_Telephone_vol_20d", "SPY_zscore_60d", "spx_momentum_3d", "WTI_Oil_FRED_zscore_60d", "ENB_EnbridgeInc_ret_1d", "LOW_Lowes_ret_20d", "AMD_ret_5d", "AMGN_Amgen_ret_1d", "AMD_ret_1d", "HangSeng_HK_ret_5d", "NEE_NextEra_ret_20d"], "is_new": true}, {"model_id": "new_h10_NORMAL_LogisticRegression_N20_t3", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 10, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "GE_ret_1d", "Industrial_Production_zscore_60d", "hmm_p_stress", "SCHW_Schwab_ret_5d", "SLB_Schlumberger_ret_5d", "gjr_condvar_h1", "US6M_Rate_ret_20d", "MS_MorganStanley_zscore_60d", "EWM_Malaysia_vol_20d", "EFFR_vol_20d", "AXP_Amex_vol_20d", "Nikkei_Japan_vol_20d", "PFE_ret_1d", "Retail_Sales_zscore_60d", "CPB_CampbellSoup_ret_20d", "LOW_Lowes_ret_5d", "AMT_AmericanTower_ret_1d", "GILD_Gilead_ret_20d"], "is_new": true}, {"model_id": "new_h10_NORMAL_LogisticRegression_N20_t4", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 10, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "XLF_Fin_vol_20d", "DAX_Germany_zscore_60d", "ENB_EnbridgeInc_ret_1d", "EMR_Emerson_ret_20d", "INTC_ret_5d", "IWM_SmallCap_vol_20d", "EWA_Australia_ret_1d", "XLK_Tech_zscore_60d", "SO_SouthernCo_ret_5d", "EWM_Malaysia_ret_1d", "BTI_BritishAmerican_ret_20d", "LMT_LockheedMartin_ret_1d", "AMD_ret_1d", "AVB_AvalonBay_zscore_60d", "PPL_PPL_ret_1d", "gjr_condvar_h1", "MSTR_Bitcoin3_ret_5d", "EWJ_Japan_vol_20d"], "is_new": true}, {"model_id": "new_h10_NORMAL_LogisticRegression_N20_t5", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 10, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EQR_Equity_ret_1d", "MSTR_Bitcoin3_ret_20d", "ENB_EnbridgeInc_ret_1d", "TED_Spread_zscore_60d", "LMT_LockheedMartin_ret_1d", "PCAR_PaccarInc_ret_5d", "EWQ_France_ret_20d", "PAYX_Paychex_vol_20d", "INTC_ret_5d", "MS_MorganStanley_ret_5d", "SO_SouthernCo_ret_5d", "GE_ret_1d", "NEE_NextEra_ret_20d", "XLF_Fin_vol_20d", "Michigan_Sentiment_ret_20d", "DE_Deere_vol_20d", "LOW_Lowes_ret_20d", "CI_Cigna_vol_20d"], "is_new": true}, {"model_id": "new_h10_NORMAL_LogisticRegression_N20_t6", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 10, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "DIS_vol_20d", "HD_zscore_60d", "3M_ret_5d", "INTC_ret_1d", "spx_abs_ret_max_5d", "Brent_Oil_FRED_ret_5d", "NWL_Newell_ret_20d", "EFFR_vol_20d", "HD_ret_20d", "EWQ_France_zscore_60d", "PG_ret_20d", "CPB_CampbellSoup_ret_5d", "VVIX_ret_20d", "MO_AltriaMG_ret_1d", "XLF_Fin_vol_20d", "EWC_Canada_zscore_60d", "DAX_Germany_zscore_60d", "T_ret_1d"], "is_new": true}, {"model_id": "new_h10_NORMAL_LogisticRegression_N20_t7", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 10, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "NFCI_ret_5d", "BA_ret_1d", "EFFR_ret_1d", "SLB_Schlumberger_ret_1d", "spx_momentum_3d", "PPL_PPL_ret_1d", "US6M_Rate_ret_20d", "AMT_AmericanTower_ret_1d", "DIS_vol_20d", "EWM_Malaysia_vol_20d", "DAX_Germany_vol_20d", "EXC_Exelon_zscore_60d", "US3Y_Rate_ret_5d", "VVIX_ret_20d", "MO_AltriaMG_ret_1d", "CPB_CampbellSoup_ret_20d", "PG_ret_20d", "QQQ_vol_20d"], "is_new": true}, {"model_id": "new_h10_NORMAL_LogisticRegression_N25_t0", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 10, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "AORD_AUS_zscore_60d", "DAX_Germany_zscore_60d", "EWQ_France_zscore_60d", "CI_Cigna_vol_20d", "MS_MorganStanley_zscore_60d", "EFFR_vol_20d", "TM_Telephone_vol_20d", "EWL_Switzerland_zscore_60d", "ES_Evergy_ret_1d", "XOM_ret_1d", "TGT_Target_zscore_60d", "EWM_Malaysia_vol_20d", "CPB_CampbellSoup_zscore_60d", "EOG_EOGResources_vol_20d", "EWS_Singapore_ret_5d", "Industrial_Production_zscore_60d", "US1Y_Rate_ret_20d", "LOW_Lowes_ret_20d", "DE_Deere_vol_20d", "AXP_Amex_vol_20d", "heston_var_ev_h7", "BTI_BritishAmerican_ret_5d", "VOD_Vodafone_zscore_60d"], "is_new": true}, {"model_id": "new_h10_NORMAL_LogisticRegression_N25_t1", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 10, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "AMT_AmericanTower_ret_1d", "SBUX_zscore_60d", "INTC_ret_5d", "T_ret_1d", "AMGN_Amgen_ret_1d", "SLB_Schlumberger_ret_1d", "MO_AltriaMG_ret_1d", "XLB_Materials_zscore_60d", "VVIX_ret_20d", "SBUX_ret_5d", "EWQ_France_ret_20d", "LUV_SouthwestAir_ret_5d", "US30Y_Rate_ret_20d", "T10Y2Y_Spread_ret_5d", "DHR_ret_1d", "HD_ret_1d", "HUM_Humana_ret_5d", "CPB_CampbellSoup_zscore_60d", "ORCL_vol_20d", "Retail_Sales_zscore_60d", "DE_Deere_ret_5d", "SJM_JM_Smucker_ret_5d", "BTI_BritishAmerican_ret_5d"], "is_new": true}, {"model_id": "new_h10_NORMAL_LogisticRegression_N25_t2", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 10, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWM_Malaysia_vol_20d", "LMT_LockheedMartin_vol_20d", "PG_ret_20d", "EQR_Equity_ret_1d", "DAX_Germany_zscore_60d", "ITT_ITTInc_ret_5d", "PAYX_Paychex_vol_20d", "HD_ret_5d", "DHR_vol_20d", "EWG_Germany_ret_20d", "MS_MorganStanley_ret_5d", "PAYX_Paychex_ret_20d", "LUV_SouthwestAir_ret_5d", "SLB_Schlumberger_ret_1d", "AMT_AmericanTower_ret_1d", "vix_acceleration_1d", "INTC_ret_5d", "EOG_EOGResources_ret_5d", "ASX_Australia_vol_20d", "EXC_Exelon_ret_1d", "CPB_CampbellSoup_ret_20d", "US6M_Rate_ret_20d", "PCAR_PaccarInc_ret_5d"], "is_new": true}, {"model_id": "new_h10_NORMAL_LogisticRegression_N25_t3", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 10, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "NVDA_vol_20d", "AMD_ret_5d", "CTAS_Cintas_vol_20d", "Brent_Oil_FRED_ret_5d", "spx_abs_ret_max_5d", "AMD_ret_1d", "SJM_JM_Smucker_ret_5d", "LUV_SouthwestAir_ret_5d", "DOW_Price_zscore_60d", "TXN_vol_20d", "BTI_BritishAmerican_ret_5d", "US3Y_Rate_ret_5d", "HD_ret_5d", "MSTR_Bitcoin3_ret_5d", "INTC_ret_1d", "WTI_Oil_FRED_zscore_60d", "HangSeng_HK_vol_20d", "3M_vol_20d", "DHR_vol_20d", "BTI_BritishAmerican_ret_20d", "MSTR_Bitcoin3_ret_20d", "Brent_Oil_FRED_ret_20d", "GILD_Gilead_ret_20d"], "is_new": true}, {"model_id": "new_h10_NORMAL_LogisticRegression_N25_t4", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 10, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "US5Y_Rate_ret_5d", "CPB_CampbellSoup_ret_5d", "EQR_Equity_ret_1d", "MO_AltriaMG_ret_1d", "MSTR_Bitcoin3_ret_5d", "VRP_ma5", "spx_momentum_3d", "DIS_vol_20d", "INTC_ret_1d", "XOM_ret_1d", "Nikkei_Japan_vol_20d", "EWJ_Japan_vol_20d", "CCI_CrownCastle_vol_20d", "SCHW_Schwab_ret_5d", "NFCI_ret_5d", "Core_CPI_zscore_60d", "MS_MorganStanley_zscore_60d", "XLY_Disc_vol_20d", "IBEX_Spain_ret_20d", "CLX_Clorox_vol_20d", "FedFunds_zscore_60d", "SPY_zscore_60d", "EWM_Malaysia_zscore_60d"], "is_new": true}, {"model_id": "new_h10_NORMAL_LogisticRegression_N25_t5", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 10, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "LOW_Lowes_ret_20d", "DHR_ret_1d", "NOC_Northrop_ret_20d", "BLK_BlackRock_zscore_60d", "MS_MorganStanley_zscore_60d", "IBEX_Spain_ret_20d", "EWY_Korea_zscore_60d", "XLF_Fin_vol_20d", "ASX_Australia_ret_5d", "AXP_Amex_ret_20d", "EWM_Malaysia_ret_1d", "T_ret_1d", "XLK_Tech_zscore_60d", "CTAS_Cintas_vol_20d", "AVB_AvalonBay_zscore_60d", "PAYX_Paychex_zscore_60d", "HD_ret_5d", "EXC_Exelon_zscore_60d", "MSTR_Bitcoin3_ret_1d", "PAYX_Paychex_vol_20d", "LUV_SouthwestAir_ret_5d", "ORCL_vol_20d", "CCI_CrownCastle_vol_20d"], "is_new": true}, {"model_id": "new_h10_NORMAL_LogisticRegression_N25_t6", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 10, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "spx_momentum_3d", "HangSeng_HK_ret_1d", "Nikkei_Japan_zscore_60d", "US5Y_Rate_ret_5d", "spx_abs_ret_max_5d", "3M_ret_5d", "EFFR_ret_1d", "EWY_Korea_ret_20d", "heston_var_ev_h5", "AMD_ret_1d", "LLY_zscore_60d", "ENB_EnbridgeInc_ret_1d", "CCI_CrownCastle_vol_20d", "SBUX_ret_5d", "US3M_Rate_vol_20d", "LUV_SouthwestAir_ret_5d", "TXN_vol_20d", "EWH_HongKong_ret_5d", "heston_var_ev_h7", "DHR_ret_1d", "heston_ev_h3", "PLD_Prologis_ret_5d", "XLY_Disc_vol_20d"], "is_new": true}, {"model_id": "new_h10_NORMAL_LogisticRegression_N25_t7", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 10, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "DE_Deere_vol_20d", "MRK_Merck_zscore_60d", "SCHW_Schwab_ret_5d", "ASX_Australia_ret_5d", "EFFR_vol_20d", "MSTR_Bitcoin3_ret_5d", "LMT_LockheedMartin_vol_20d", "US30Y_Rate_ret_20d", "AMT_AmericanTower_ret_1d", "CPB_CampbellSoup_vol_20d", "PCAR_PaccarInc_ret_5d", "IBEX_Spain_ret_20d", "US1Y_Rate_ret_20d", "NEE_NextEra_ret_20d", "EWH_HongKong_ret_5d", "SPY_zscore_60d", "ORCL_zscore_60d", "TED_Spread_vol_20d", "Industrial_Production_zscore_60d", "EWA_Australia_ret_1d", "TM_Telephone_vol_20d", "AXP_Amex_ret_20d", "MS_MorganStanley_zscore_60d"], "is_new": true}, {"model_id": "new_h10_NORMAL_LogisticRegression_N30_t0", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 10, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "XLF_Fin_vol_20d", "NVDA_vol_20d", "NFCI_ret_5d", "TGT_Target_zscore_60d", "HangSeng_HK_vol_20d", "EFFR_ret_1d", "WTI_Oil_FRED_zscore_60d", "DIS_vol_20d", "EWQ_France_zscore_60d", "US3M_Rate_vol_20d", "Brent_Oil_FRED_ret_5d", "EOG_EOGResources_vol_20d", "EOG_EOGResources_ret_5d", "EWA_Australia_ret_1d", "IYM_BasicMaterials_ret_20d", "Michigan_Sentiment_ret_20d", "EWJ_Japan_vol_20d", "BDX_Becton_Dickinson_ret_20d", "PPL_PPL_ret_1d", "LOW_Lowes_ret_20d", "LOW_Lowes_ret_5d", "SBUX_vol_20d", "AORD_AUS_zscore_60d", "MS_MorganStanley_ret_5d", "EWH_HongKong_ret_5d", "SLB_Schlumberger_ret_5d", "AVB_AvalonBay_zscore_60d", "XLK_Tech_zscore_60d"], "is_new": true}, {"model_id": "new_h10_NORMAL_LogisticRegression_N30_t1", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 10, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWG_Germany_vol_20d", "US3Y_Rate_ret_5d", "CMCSA_ret_1d", "Core_CPI_zscore_60d", "vix_acceleration_1d", "SO_SouthernCo_ret_5d", "EWC_Canada_zscore_60d", "EWM_Malaysia_ret_1d", "EQIX_Equinix_ret_5d", "HangSeng_HK_ret_5d", "T10Y2Y_Spread_ret_5d", "DIS_vol_20d", "heston_var_ev_h5", "ORCL_vol_20d", "US1Y_Rate_ret_20d", "gjr_condvar_h1", "vix_mean_abs_ret_5d", "BTI_BritishAmerican_ret_5d", "XOM_ret_20d", "SLB_Schlumberger_ret_5d", "SLB_Schlumberger_ret_1d", "heston_var_ev_h3", "US7Y_Rate_ret_20d", "TED_Spread_vol_20d", "EWS_Singapore_ret_5d", "DHR_vol_20d", "AMD_ret_5d", "GD_GeneralDynamics_zscore_60d"], "is_new": true}, {"model_id": "new_h10_NORMAL_LogisticRegression_N30_t2", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 10, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "HangSeng_HK_vol_20d", "SBUX_vol_20d", "HD_ret_5d", "EXC_Exelon_ret_1d", "TGT_Target_zscore_60d", "ORCL_vol_20d", "TM_Telephone_ret_1d", "XLK_Tech_zscore_60d", "NWL_Newell_ret_20d", "US1Y_Rate_ret_20d", "EWM_Malaysia_ret_1d", "XLB_Materials_zscore_60d", "EOG_EOGResources_ret_5d", "US3Y_Rate_ret_5d", "DHR_vol_20d", "MS_MorganStanley_ret_1d", "SBUX_zscore_60d", "Core_PCE_zscore_60d", "EWA_Australia_zscore_60d", "EXC_Exelon_zscore_60d", "SLB_Schlumberger_ret_1d", "heston_ev_h3", "CPB_CampbellSoup_vol_20d", "IBEX_Spain_ret_20d", "T10Y2Y_Spread_ret_5d", "HUM_Humana_ret_5d", "GILD_Gilead_ret_20d", "EOG_EOGResources_vol_20d"], "is_new": true}, {"model_id": "new_h10_NORMAL_LogisticRegression_N30_t3", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 10, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWL_Switzerland_vol_20d", "TGT_Target_zscore_60d", "VRP_ma5", "EWG_Germany_ret_20d", "NOC_Northrop_ret_20d", "SBUX_ret_5d", "BLK_BlackRock_zscore_60d", "3M_ret_5d", "EWA_Australia_zscore_60d", "spx_abs_ret_max_5d", "US3Y_Rate_ret_5d", "AMT_AmericanTower_ret_1d", "EWY_Korea_zscore_60d", "EOG_EOGResources_ret_5d", "MSTR_Bitcoin3_ret_20d", "BDX_Becton_Dickinson_ret_20d", "IYM_BasicMaterials_ret_20d", "ASX_Australia_vol_20d", "LUV_SouthwestAir_ret_5d", "PPL_PPL_ret_1d", "ENB_EnbridgeInc_ret_1d", "XLF_Fin_vol_20d", "CI_Cigna_vol_20d", "NEE_NextEra_ret_20d", "SJM_JM_Smucker_ret_5d", "GILD_Gilead_ret_20d", "EXC_Exelon_zscore_60d", "PCAR_PaccarInc_ret_5d"], "is_new": true}, {"model_id": "new_h10_NORMAL_LogisticRegression_N30_t4", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 10, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "NVDA_vol_20d", "US1Y_Rate_ret_5d", "ENB_EnbridgeInc_ret_1d", "LMT_LockheedMartin_ret_1d", "SJM_JM_Smucker_ret_1d", "BA_ret_1d", "Nikkei_Japan_vol_20d", "ITT_ITTInc_ret_5d", "vix_mean_abs_ret_5d", "EWS_Singapore_ret_5d", "EOG_EOGResources_vol_20d", "AORD_AUS_zscore_60d", "AMZN_ret_5d", "EWL_Switzerland_vol_20d", "PCAR_PaccarInc_ret_5d", "HUM_Humana_ret_5d", "LMT_LockheedMartin_vol_20d", "DHR_vol_20d", "BLK_BlackRock_zscore_60d", "CPB_CampbellSoup_ret_5d", "HD_ret_1d", "SBUX_zscore_60d", "AXP_Amex_vol_20d", "vix_acceleration_1d", "CCI_CrownCastle_vol_20d", "LLY_zscore_60d", "SLB_Schlumberger_ret_5d", "XLY_Disc_vol_20d"], "is_new": true}, {"model_id": "new_h10_NORMAL_LogisticRegression_N30_t5", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 10, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EMR_Emerson_ret_20d", "PFE_ret_1d", "CCI_CrownCastle_vol_20d", "ORCL_zscore_60d", "EOG_EOGResources_ret_5d", "HangSeng_HK_ret_5d", "EXC_Exelon_ret_1d", "EFFR_ret_1d", "DE_Deere_vol_20d", "EWQ_France_ret_20d", "DAX_Germany_zscore_60d", "EWS_Singapore_ret_5d", "MO_AltriaMG_ret_1d", "spx_abs_ret_max_5d", "PAYX_Paychex_vol_20d", "US3Y_Rate_ret_5d", "BTI_BritishAmerican_ret_20d", "MSTR_Bitcoin3_ret_5d", "ASX_Australia_vol_20d", "BA_ret_1d", "LLY_zscore_60d", "EWJ_Japan_vol_20d", "EWA_Australia_ret_1d", "BDX_Becton_Dickinson_ret_20d", "AMGN_Amgen_ret_1d", "LOW_Lowes_ret_5d", "JNJ_ret_1d", "M_Macys_vol_20d"], "is_new": true}, {"model_id": "new_h10_NORMAL_LogisticRegression_N30_t6", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 10, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "TED_Spread_zscore_60d", "INTC_ret_5d", "spx_momentum_3d", "ASX_Australia_ret_5d", "VOD_Vodafone_zscore_60d", "US7Y_Rate_ret_20d", "DE_Deere_vol_20d", "US5Y_Rate_ret_5d", "3M_vol_20d", "Brent_Oil_FRED_ret_20d", "3M_ret_5d", "IYR_US_REIT2_zscore_60d", "hmm_p_stress", "ORCL_vol_20d", "PCAR_PaccarInc_ret_5d", "TM_Telephone_ret_1d", "US30Y_Rate_ret_20d", "AXP_Amex_vol_20d", "Nikkei_Japan_zscore_60d", "DHR_vol_20d", "INTC_ret_1d", "EOG_EOGResources_vol_20d", "US3M_Rate_zscore_60d", "CLX_Clorox_vol_20d", "SLB_Schlumberger_ret_1d", "PAYX_Paychex_ret_20d", "ORCL_zscore_60d", "EWH_HongKong_ret_5d"], "is_new": true}, {"model_id": "new_h10_NORMAL_LogisticRegression_N30_t7", "algo": "LogisticRegression", "regime": "NORMAL", "horizon": 10, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWL_Switzerland_vol_20d", "Retail_Sales_zscore_60d", "SCHW_Schwab_ret_5d", "ASX_Australia_vol_20d", "EWY_Korea_ret_20d", "DIS_vol_20d", "US3Y_Rate_ret_5d", "LOW_Lowes_ret_20d", "CLX_Clorox_vol_20d", "QQQ_vol_20d", "AMD_ret_1d", "heston_ev_h3", "NFCI_ret_5d", "PAYX_Paychex_ret_20d", "PFE_ret_1d", "LMT_LockheedMartin_ret_1d", "heston_var_ev_h7", "MO_AltriaMG_ret_1d", "SJM_JM_Smucker_ret_1d", "PG_ret_20d", "3M_ret_5d", "IBEX_Spain_ret_20d", "TED_Spread_vol_20d", "EWQ_France_zscore_60d", "EOG_EOGResources_ret_5d", "CPB_CampbellSoup_ret_20d", "LUV_SouthwestAir_ret_5d", "NWL_Newell_ret_20d"], "is_new": true}, {"model_id": "new_h10_STRESS_XGBoost_N5_t0", "algo": "XGBoost", "regime": "STRESS", "horizon": 10, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWS_Singapore_ret_5d", "vix_mean_abs_ret_5d", "XLV_Health_zscore_60d"], "is_new": true}, {"model_id": "new_h10_STRESS_XGBoost_N5_t1", "algo": "XGBoost", "regime": "STRESS", "horizon": 10, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "DOW_Price_zscore_60d", "MS_MorganStanley_ret_1d", "heston_ev_h3"], "is_new": true}, {"model_id": "new_h10_STRESS_XGBoost_N5_t2", "algo": "XGBoost", "regime": "STRESS", "horizon": 10, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "MSTR_Bitcoin3_ret_1d", "EWA_Australia_ret_1d", "Core_CPI_zscore_60d"], "is_new": true}, {"model_id": "new_h10_STRESS_XGBoost_N5_t3", "algo": "XGBoost", "regime": "STRESS", "horizon": 10, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "IBEX_Spain_ret_20d", "EWY_Korea_zscore_60d", "vix_acceleration_1d"], "is_new": true}, {"model_id": "new_h10_STRESS_XGBoost_N5_t4", "algo": "XGBoost", "regime": "STRESS", "horizon": 10, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "spx_momentum_3d", "DHR_vol_20d", "spx_abs_ret_max_5d"], "is_new": true}, {"model_id": "new_h10_STRESS_XGBoost_N5_t5", "algo": "XGBoost", "regime": "STRESS", "horizon": 10, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "HangSeng_HK_vol_20d", "MS_MorganStanley_ret_5d", "US1Y_Rate_ret_5d"], "is_new": true}, {"model_id": "new_h10_STRESS_XGBoost_N5_t6", "algo": "XGBoost", "regime": "STRESS", "horizon": 10, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "DHR_vol_20d", "LMT_LockheedMartin_ret_1d", "CPB_CampbellSoup_zscore_60d"], "is_new": true}, {"model_id": "new_h10_STRESS_XGBoost_N5_t7", "algo": "XGBoost", "regime": "STRESS", "horizon": 10, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWM_Malaysia_vol_20d", "HangSeng_HK_ret_5d", "EWY_Korea_zscore_60d"], "is_new": true}, {"model_id": "new_h10_STRESS_XGBoost_N8_t0", "algo": "XGBoost", "regime": "STRESS", "horizon": 10, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "AMD_ret_1d", "Core_PCE_zscore_60d", "EWM_Malaysia_vol_20d", "ENB_EnbridgeInc_ret_1d", "vix_acceleration_1d", "SBUX_zscore_60d"], "is_new": true}, {"model_id": "new_h10_STRESS_XGBoost_N8_t1", "algo": "XGBoost", "regime": "STRESS", "horizon": 10, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "PAYX_Paychex_vol_20d", "CCI_CrownCastle_vol_20d", "HD_ret_5d", "DIS_vol_20d", "HUM_Humana_ret_5d", "HangSeng_HK_vol_20d"], "is_new": true}, {"model_id": "new_h10_STRESS_XGBoost_N8_t2", "algo": "XGBoost", "regime": "STRESS", "horizon": 10, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "PFE_ret_1d", "PLD_Prologis_ret_5d", "AMT_AmericanTower_ret_1d", "EXC_Exelon_ret_1d", "INTC_ret_5d", "TGT_Target_zscore_60d"], "is_new": true}, {"model_id": "new_h10_STRESS_XGBoost_N8_t3", "algo": "XGBoost", "regime": "STRESS", "horizon": 10, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "gjr_condvar_h1", "EWY_Korea_zscore_60d", "MSTR_Bitcoin3_ret_5d", "BA_ret_1d", "DHR_vol_20d", "Michigan_Sentiment_ret_20d"], "is_new": true}, {"model_id": "new_h10_STRESS_XGBoost_N8_t4", "algo": "XGBoost", "regime": "STRESS", "horizon": 10, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "TED_Spread_vol_20d", "ENB_EnbridgeInc_ret_1d", "CLX_Clorox_vol_20d", "EWA_Australia_zscore_60d", "EWG_Germany_vol_20d", "EWY_Korea_zscore_60d"], "is_new": true}, {"model_id": "new_h10_STRESS_XGBoost_N8_t5", "algo": "XGBoost", "regime": "STRESS", "horizon": 10, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "HD_ret_1d", "US30Y_Rate_ret_20d", "EWS_Singapore_ret_5d", "HangSeng_HK_ret_5d", "US3M_Rate_zscore_60d", "Core_CPI_zscore_60d"], "is_new": true}, {"model_id": "new_h10_STRESS_XGBoost_N8_t6", "algo": "XGBoost", "regime": "STRESS", "horizon": 10, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "GILD_Gilead_ret_20d", "PG_ret_20d", "GD_GeneralDynamics_zscore_60d", "US5Y_Rate_ret_5d", "EOG_EOGResources_vol_20d", "ORCL_vol_20d"], "is_new": true}, {"model_id": "new_h10_STRESS_XGBoost_N8_t7", "algo": "XGBoost", "regime": "STRESS", "horizon": 10, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EQIX_Equinix_ret_5d", "XOM_ret_1d", "AXP_Amex_vol_20d", "TED_Spread_vol_20d", "US5Y_Rate_ret_5d", "TXN_vol_20d"], "is_new": true}, {"model_id": "new_h10_STRESS_XGBoost_N10_t0", "algo": "XGBoost", "regime": "STRESS", "horizon": 10, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "Retail_Sales_zscore_60d", "Brent_Oil_FRED_ret_5d", "SO_SouthernCo_ret_5d", "EWL_Switzerland_zscore_60d", "HangSeng_HK_vol_20d", "EOG_EOGResources_ret_5d", "CLX_Clorox_vol_20d", "BTI_BritishAmerican_ret_5d"], "is_new": true}, {"model_id": "new_h10_STRESS_XGBoost_N10_t1", "algo": "XGBoost", "regime": "STRESS", "horizon": 10, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "DHR_vol_20d", "EXC_Exelon_zscore_60d", "US30Y_Rate_ret_20d", "AMGN_Amgen_ret_1d", "NEE_NextEra_ret_20d", "vix_acceleration_1d", "LLY_zscore_60d", "MS_MorganStanley_ret_5d"], "is_new": true}, {"model_id": "new_h10_STRESS_XGBoost_N10_t2", "algo": "XGBoost", "regime": "STRESS", "horizon": 10, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "T_ret_1d", "HD_zscore_60d", "AMZN_ret_5d", "US3M_Rate_zscore_60d", "EWL_Switzerland_zscore_60d", "TM_Telephone_vol_20d", "AMD_ret_5d", "Brent_Oil_FRED_ret_20d"], "is_new": true}, {"model_id": "new_h10_STRESS_XGBoost_N10_t3", "algo": "XGBoost", "regime": "STRESS", "horizon": 10, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "AMD_ret_5d", "CTAS_Cintas_vol_20d", "SLB_Schlumberger_ret_1d", "EWJ_Japan_vol_20d", "FedFunds_zscore_60d", "HangSeng_HK_ret_5d", "BLK_BlackRock_zscore_60d", "XLV_Health_zscore_60d"], "is_new": true}, {"model_id": "new_h10_STRESS_XGBoost_N10_t4", "algo": "XGBoost", "regime": "STRESS", "horizon": 10, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "HD_ret_20d", "LUV_SouthwestAir_ret_5d", "hmm_p_stress", "AORD_AUS_zscore_60d", "DE_Deere_vol_20d", "BDX_Becton_Dickinson_ret_20d", "WTI_Oil_FRED_zscore_60d", "EQR_Equity_ret_1d"], "is_new": true}, {"model_id": "new_h10_STRESS_XGBoost_N10_t5", "algo": "XGBoost", "regime": "STRESS", "horizon": 10, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "PFE_ret_1d", "BLK_BlackRock_zscore_60d", "CPB_CampbellSoup_vol_20d", "FedFunds_zscore_60d", "CTAS_Cintas_vol_20d", "DE_Deere_ret_5d", "hmm_p_stress", "PAYX_Paychex_vol_20d"], "is_new": true}, {"model_id": "new_h10_STRESS_XGBoost_N10_t6", "algo": "XGBoost", "regime": "STRESS", "horizon": 10, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "SO_SouthernCo_ret_5d", "CPB_CampbellSoup_ret_5d", "ASX_Australia_ret_5d", "XLF_Fin_vol_20d", "BA_ret_1d", "MS_MorganStanley_ret_1d", "EQR_Equity_ret_1d", "3M_ret_5d"], "is_new": true}, {"model_id": "new_h10_STRESS_XGBoost_N10_t7", "algo": "XGBoost", "regime": "STRESS", "horizon": 10, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "AXP_Amex_vol_20d", "ORCL_vol_20d", "EWQ_France_zscore_60d", "EWY_Korea_zscore_60d", "GILD_Gilead_ret_20d", "EWS_Singapore_ret_5d", "Nikkei_Japan_vol_20d", "DOW_Price_zscore_60d"], "is_new": true}, {"model_id": "new_h10_STRESS_XGBoost_N12_t0", "algo": "XGBoost", "regime": "STRESS", "horizon": 10, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "TM_Telephone_vol_20d", "Core_CPI_zscore_60d", "CPB_CampbellSoup_ret_20d", "spx_momentum_3d", "CI_Cigna_vol_20d", "EFFR_vol_20d", "US5Y_Rate_ret_5d", "SBUX_ret_5d", "NWL_Newell_ret_20d", "EWY_Korea_zscore_60d"], "is_new": true}, {"model_id": "new_h10_STRESS_XGBoost_N12_t1", "algo": "XGBoost", "regime": "STRESS", "horizon": 10, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "hmm_p_stress", "Industrial_Production_zscore_60d", "CPB_CampbellSoup_zscore_60d", "IYM_BasicMaterials_ret_20d", "TGT_Target_zscore_60d", "MO_AltriaMG_ret_1d", "EWM_Malaysia_vol_20d", "EWJ_Japan_vol_20d", "BTI_BritishAmerican_ret_20d", "INTC_ret_5d"], "is_new": true}, {"model_id": "new_h10_STRESS_XGBoost_N12_t2", "algo": "XGBoost", "regime": "STRESS", "horizon": 10, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "NEE_NextEra_ret_20d", "T_ret_1d", "TXN_vol_20d", "CI_Cigna_vol_20d", "EWC_Canada_zscore_60d", "ASX_Australia_vol_20d", "CPB_CampbellSoup_vol_20d", "AMD_ret_1d", "AXP_Amex_ret_20d", "TED_Spread_vol_20d"], "is_new": true}, {"model_id": "new_h10_STRESS_XGBoost_N12_t3", "algo": "XGBoost", "regime": "STRESS", "horizon": 10, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWQ_France_ret_20d", "NEE_NextEra_ret_20d", "EWG_Germany_vol_20d", "heston_var_ev_h3", "PFE_ret_1d", "AORD_AUS_zscore_60d", "EWL_Switzerland_vol_20d", "NFCI_ret_5d", "CMCSA_ret_1d", "Nikkei_Japan_zscore_60d"], "is_new": true}, {"model_id": "new_h10_STRESS_XGBoost_N12_t4", "algo": "XGBoost", "regime": "STRESS", "horizon": 10, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "US5Y_Rate_ret_5d", "EWS_Singapore_ret_5d", "Nikkei_Japan_zscore_60d", "MS_MorganStanley_ret_5d", "EFFR_ret_1d", "US3Y_Rate_ret_5d", "EWH_HongKong_ret_5d", "XLK_Tech_zscore_60d", "vix_mean_abs_ret_5d", "XOM_ret_20d"], "is_new": true}, {"model_id": "new_h10_STRESS_XGBoost_N12_t5", "algo": "XGBoost", "regime": "STRESS", "horizon": 10, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "heston_ev_h3", "TGT_Target_zscore_60d", "AMT_AmericanTower_ret_1d", "DAX_Germany_zscore_60d", "AXP_Amex_ret_20d", "SLB_Schlumberger_ret_5d", "IYM_BasicMaterials_ret_20d", "hmm_p_stress", "EQIX_Equinix_ret_5d", "MRK_Merck_zscore_60d"], "is_new": true}, {"model_id": "new_h10_STRESS_XGBoost_N12_t6", "algo": "XGBoost", "regime": "STRESS", "horizon": 10, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "PFE_ret_1d", "VVIX_ret_20d", "WTI_Oil_FRED_zscore_60d", "CPB_CampbellSoup_ret_20d", "SBUX_ret_5d", "EMR_Emerson_ret_20d", "PG_ret_20d", "SPY_zscore_60d", "AMZN_ret_5d", "EQIX_Equinix_ret_5d"], "is_new": true}, {"model_id": "new_h10_STRESS_XGBoost_N12_t7", "algo": "XGBoost", "regime": "STRESS", "horizon": 10, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "Brent_Oil_FRED_ret_5d", "US7Y_Rate_ret_20d", "TED_Spread_vol_20d", "PCAR_PaccarInc_ret_5d", "MO_AltriaMG_ret_1d", "MSTR_Bitcoin3_ret_1d", "DHR_vol_20d", "EFFR_ret_1d", "Industrial_Production_zscore_60d", "ORCL_vol_20d"], "is_new": true}, {"model_id": "new_h10_STRESS_XGBoost_N15_t0", "algo": "XGBoost", "regime": "STRESS", "horizon": 10, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "CCI_CrownCastle_vol_20d", "GE_ret_1d", "EFFR_ret_1d", "MSTR_Bitcoin3_ret_5d", "Retail_Sales_zscore_60d", "SJM_JM_Smucker_ret_1d", "HD_ret_20d", "TED_Spread_vol_20d", "spx_vol_5d", "ORCL_zscore_60d", "CPB_CampbellSoup_vol_20d", "Brent_Oil_FRED_ret_20d", "AMZN_ret_5d"], "is_new": true}, {"model_id": "new_h10_STRESS_XGBoost_N15_t1", "algo": "XGBoost", "regime": "STRESS", "horizon": 10, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "AMGN_Amgen_ret_1d", "SJM_JM_Smucker_ret_5d", "heston_var_ev_h5", "LUV_SouthwestAir_ret_5d", "T_ret_1d", "ENB_EnbridgeInc_ret_1d", "GD_GeneralDynamics_zscore_60d", "EWM_Malaysia_ret_1d", "AXP_Amex_ret_20d", "INTC_ret_1d", "CCI_CrownCastle_vol_20d", "SPY_zscore_60d", "MS_MorganStanley_ret_5d"], "is_new": true}, {"model_id": "new_h10_STRESS_XGBoost_N15_t2", "algo": "XGBoost", "regime": "STRESS", "horizon": 10, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWG_Germany_ret_20d", "MS_MorganStanley_zscore_60d", "LMT_LockheedMartin_vol_20d", "US3M_Rate_zscore_60d", "Retail_Sales_zscore_60d", "MO_AltriaMG_ret_1d", "CTAS_Cintas_vol_20d", "3M_vol_20d", "MS_MorganStanley_ret_1d", "DE_Deere_vol_20d", "XLK_Tech_zscore_60d", "EOG_EOGResources_ret_5d", "DE_Deere_ret_5d"], "is_new": true}, {"model_id": "new_h10_STRESS_XGBoost_N15_t3", "algo": "XGBoost", "regime": "STRESS", "horizon": 10, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "VVIX_ret_20d", "DHR_vol_20d", "EQIX_Equinix_ret_5d", "heston_var_ev_h7", "CLX_Clorox_vol_20d", "EWM_Malaysia_vol_20d", "Nikkei_Japan_vol_20d", "BDX_Becton_Dickinson_ret_20d", "Industrial_Production_zscore_60d", "QQQ_vol_20d", "vix_mean_abs_ret_5d", "XLK_Tech_zscore_60d", "PCAR_PaccarInc_ret_5d"], "is_new": true}, {"model_id": "new_h10_STRESS_XGBoost_N15_t4", "algo": "XGBoost", "regime": "STRESS", "horizon": 10, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "DE_Deere_ret_5d", "EWM_Malaysia_zscore_60d", "SCHW_Schwab_ret_5d", "gjr_condvar_h1", "DE_Deere_vol_20d", "XOM_ret_1d", "IWM_SmallCap_vol_20d", "EWC_Canada_zscore_60d", "FedFunds_zscore_60d", "EWL_Switzerland_vol_20d", "EMR_Emerson_ret_20d", "DHR_vol_20d", "AMZN_ret_5d"], "is_new": true}, {"model_id": "new_h10_STRESS_XGBoost_N15_t5", "algo": "XGBoost", "regime": "STRESS", "horizon": 10, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "XLK_Tech_zscore_60d", "GILD_Gilead_ret_20d", "HD_ret_1d", "MRK_Merck_zscore_60d", "PCAR_PaccarInc_ret_5d", "DOW_Price_zscore_60d", "US30Y_Rate_ret_20d", "EFFR_vol_20d", "MS_MorganStanley_ret_1d", "Core_CPI_zscore_60d", "SLB_Schlumberger_ret_1d", "NFCI_ret_5d", "QQQ_vol_20d"], "is_new": true}, {"model_id": "new_h10_STRESS_XGBoost_N15_t6", "algo": "XGBoost", "regime": "STRESS", "horizon": 10, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "SBUX_zscore_60d", "DE_Deere_ret_5d", "LLY_zscore_60d", "US3M_Rate_zscore_60d", "US3M_Rate_vol_20d", "SLB_Schlumberger_ret_1d", "HD_ret_20d", "XLB_Materials_zscore_60d", "US1Y_Rate_ret_20d", "MSTR_Bitcoin3_ret_20d", "3M_ret_5d", "CCI_CrownCastle_vol_20d", "SJM_JM_Smucker_ret_1d"], "is_new": true}, {"model_id": "new_h10_STRESS_XGBoost_N15_t7", "algo": "XGBoost", "regime": "STRESS", "horizon": 10, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "DOW_Price_zscore_60d", "EMR_Emerson_ret_20d", "DAX_Germany_vol_20d", "PAYX_Paychex_vol_20d", "Brent_Oil_FRED_ret_20d", "HUM_Humana_ret_5d", "NWL_Newell_ret_20d", "SJM_JM_Smucker_ret_1d", "FedFunds_zscore_60d", "hmm_p_stress", "Core_PCE_zscore_60d", "TM_Telephone_vol_20d", "AMT_AmericanTower_ret_1d"], "is_new": true}, {"model_id": "new_h10_STRESS_XGBoost_N20_t0", "algo": "XGBoost", "regime": "STRESS", "horizon": 10, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "TM_Telephone_vol_20d", "WTI_Oil_FRED_zscore_60d", "Core_PCE_zscore_60d", "GILD_Gilead_ret_20d", "Michigan_Sentiment_ret_20d", "EFFR_vol_20d", "CPB_CampbellSoup_ret_20d", "spx_abs_ret_max_5d", "SLB_Schlumberger_ret_5d", "XLY_Disc_vol_20d", "HD_ret_5d", "CPB_CampbellSoup_ret_5d", "Retail_Sales_zscore_60d", "US3M_Rate_zscore_60d", "LMT_LockheedMartin_ret_1d", "SJM_JM_Smucker_ret_1d", "MO_AltriaMG_ret_1d", "TED_Spread_vol_20d"], "is_new": true}, {"model_id": "new_h10_STRESS_XGBoost_N20_t1", "algo": "XGBoost", "regime": "STRESS", "horizon": 10, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "Nikkei_Japan_vol_20d", "XLB_Materials_zscore_60d", "spx_momentum_3d", "AMT_AmericanTower_ret_1d", "PFE_ret_1d", "MS_MorganStanley_zscore_60d", "NEE_NextEra_ret_20d", "heston_var_ev_h5", "US3M_Rate_zscore_60d", "XLK_Tech_zscore_60d", "EWC_Canada_zscore_60d", "vix_acceleration_1d", "ORCL_vol_20d", "CTAS_Cintas_vol_20d", "EWG_Germany_ret_20d", "DIS_vol_20d", "heston_ev_h3", "PPL_PPL_ret_1d"], "is_new": true}, {"model_id": "new_h10_STRESS_XGBoost_N20_t2", "algo": "XGBoost", "regime": "STRESS", "horizon": 10, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "US3M_Rate_zscore_60d", "CI_Cigna_vol_20d", "TM_Telephone_vol_20d", "spx_vol_5d", "PG_ret_20d", "EWM_Malaysia_zscore_60d", "DOW_Price_zscore_60d", "IWM_SmallCap_vol_20d", "Brent_Oil_FRED_ret_20d", "GD_GeneralDynamics_zscore_60d", "ITT_ITTInc_ret_5d", "HD_ret_20d", "NOC_Northrop_ret_20d", "AMGN_Amgen_ret_1d", "Nikkei_Japan_vol_20d", "vix_acceleration_1d", "AXP_Amex_vol_20d", "PLD_Prologis_ret_5d"], "is_new": true}, {"model_id": "new_h10_STRESS_XGBoost_N20_t3", "algo": "XGBoost", "regime": "STRESS", "horizon": 10, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "PPL_PPL_ret_1d", "US3M_Rate_zscore_60d", "Brent_Oil_FRED_ret_20d", "SPY_zscore_60d", "XLF_Fin_vol_20d", "IBEX_Spain_ret_20d", "HangSeng_HK_ret_5d", "WTI_Oil_FRED_zscore_60d", "XLV_Health_zscore_60d", "SCHW_Schwab_ret_5d", "ASX_Australia_vol_20d", "PFE_ret_1d", "AMZN_ret_5d", "Industrial_Production_zscore_60d", "GILD_Gilead_ret_20d", "US30Y_Rate_ret_20d", "EOG_EOGResources_vol_20d", "DAX_Germany_vol_20d"], "is_new": true}, {"model_id": "new_h10_STRESS_XGBoost_N20_t4", "algo": "XGBoost", "regime": "STRESS", "horizon": 10, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "LMT_LockheedMartin_vol_20d", "US3M_Rate_vol_20d", "US6M_Rate_ret_20d", "EWA_Australia_ret_1d", "MRK_Merck_zscore_60d", "EWC_Canada_zscore_60d", "SJM_JM_Smucker_ret_5d", "HangSeng_HK_vol_20d", "NVDA_vol_20d", "CI_Cigna_vol_20d", "EOG_EOGResources_vol_20d", "heston_var_ev_h3", "JNJ_ret_1d", "IWM_SmallCap_vol_20d", "spx_vol_5d", "XLF_Fin_vol_20d", "EQIX_Equinix_ret_5d", "ES_Evergy_ret_1d"], "is_new": true}, {"model_id": "new_h10_STRESS_XGBoost_N20_t5", "algo": "XGBoost", "regime": "STRESS", "horizon": 10, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "heston_var_ev_h5", "DE_Deere_ret_5d", "TM_Telephone_ret_1d", "PPL_PPL_ret_1d", "TXN_vol_20d", "DAX_Germany_vol_20d", "Brent_Oil_FRED_ret_5d", "NWL_Newell_ret_20d", "ITT_ITTInc_ret_5d", "heston_ev_h3", "XLB_Materials_zscore_60d", "TM_Telephone_vol_20d", "heston_var_ev_h7", "EWM_Malaysia_zscore_60d", "hmm_p_stress", "US5Y_Rate_ret_5d", "CPB_CampbellSoup_vol_20d", "EWL_Switzerland_vol_20d"], "is_new": true}, {"model_id": "new_h10_STRESS_XGBoost_N20_t6", "algo": "XGBoost", "regime": "STRESS", "horizon": 10, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "heston_var_ev_h5", "EWG_Germany_ret_20d", "DIS_vol_20d", "EWM_Malaysia_vol_20d", "IYM_BasicMaterials_ret_20d", "VRP_ma5", "CCI_CrownCastle_vol_20d", "NFCI_ret_5d", "heston_ev_h3", "MSTR_Bitcoin3_ret_5d", "spx_momentum_3d", "CPB_CampbellSoup_vol_20d", "BDX_Becton_Dickinson_ret_20d", "PLD_Prologis_ret_5d", "HD_ret_5d", "HD_zscore_60d", "EWY_Korea_zscore_60d", "EWJ_Japan_vol_20d"], "is_new": true}, {"model_id": "new_h10_STRESS_XGBoost_N20_t7", "algo": "XGBoost", "regime": "STRESS", "horizon": 10, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "DAX_Germany_vol_20d", "BTI_BritishAmerican_ret_5d", "BDX_Becton_Dickinson_ret_20d", "GD_GeneralDynamics_zscore_60d", "spx_vol_5d", "US1Y_Rate_ret_20d", "Michigan_Sentiment_ret_20d", "ITT_ITTInc_ret_5d", "SCHW_Schwab_ret_5d", "TM_Telephone_ret_1d", "EQIX_Equinix_ret_5d", "XLY_Disc_vol_20d", "INTC_ret_1d", "PCAR_PaccarInc_ret_5d", "TGT_Target_zscore_60d", "SBUX_vol_20d", "Retail_Sales_zscore_60d", "IYR_US_REIT2_zscore_60d"], "is_new": true}, {"model_id": "new_h10_STRESS_XGBoost_N25_t0", "algo": "XGBoost", "regime": "STRESS", "horizon": 10, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "SO_SouthernCo_ret_5d", "Brent_Oil_FRED_ret_5d", "AMD_ret_5d", "EQIX_Equinix_ret_5d", "MRK_Merck_zscore_60d", "JNJ_ret_1d", "heston_var_ev_h7", "AVB_AvalonBay_zscore_60d", "TM_Telephone_vol_20d", "HangSeng_HK_ret_1d", "EWM_Malaysia_vol_20d", "EMR_Emerson_ret_20d", "EWC_Canada_zscore_60d", "HD_zscore_60d", "GE_ret_1d", "US6M_Rate_ret_20d", "PLD_Prologis_ret_5d", "EFFR_vol_20d", "SBUX_zscore_60d", "LMT_LockheedMartin_ret_1d", "MO_AltriaMG_ret_1d", "IWM_SmallCap_vol_20d", "Nikkei_Japan_zscore_60d"], "is_new": true}, {"model_id": "new_h10_STRESS_XGBoost_N25_t1", "algo": "XGBoost", "regime": "STRESS", "horizon": 10, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "VOD_Vodafone_zscore_60d", "CI_Cigna_vol_20d", "heston_var_ev_h5", "NOC_Northrop_ret_20d", "Brent_Oil_FRED_ret_5d", "SBUX_ret_5d", "AVB_AvalonBay_zscore_60d", "TGT_Target_zscore_60d", "GILD_Gilead_ret_20d", "ORCL_vol_20d", "AORD_AUS_zscore_60d", "XLF_Fin_vol_20d", "HUM_Humana_ret_5d", "LLY_zscore_60d", "MS_MorganStanley_zscore_60d", "SBUX_vol_20d", "LUV_SouthwestAir_ret_5d", "US1Y_Rate_ret_5d", "AMT_AmericanTower_ret_1d", "US6M_Rate_ret_20d", "IBEX_Spain_ret_20d", "TXN_vol_20d", "EWL_Switzerland_vol_20d"], "is_new": true}, {"model_id": "new_h10_STRESS_XGBoost_N25_t2", "algo": "XGBoost", "regime": "STRESS", "horizon": 10, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWG_Germany_ret_20d", "WTI_Oil_FRED_zscore_60d", "Industrial_Production_zscore_60d", "US3Y_Rate_ret_5d", "AVB_AvalonBay_zscore_60d", "PLD_Prologis_ret_5d", "spx_vol_5d", "HUM_Humana_ret_5d", "CI_Cigna_vol_20d", "Brent_Oil_FRED_ret_20d", "US30Y_Rate_ret_20d", "EWA_Australia_ret_1d", "EFFR_ret_1d", "HD_ret_20d", "Core_CPI_zscore_60d", "AMT_AmericanTower_ret_1d", "DE_Deere_ret_5d", "GE_ret_1d", "NWL_Newell_ret_20d", "heston_var_ev_h5", "AMD_ret_1d", "MSTR_Bitcoin3_ret_1d", "EWM_Malaysia_ret_1d"], "is_new": true}, {"model_id": "new_h10_STRESS_XGBoost_N25_t3", "algo": "XGBoost", "regime": "STRESS", "horizon": 10, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "XLF_Fin_vol_20d", "AVB_AvalonBay_zscore_60d", "DHR_ret_1d", "LOW_Lowes_ret_5d", "CMCSA_ret_1d", "CPB_CampbellSoup_ret_5d", "US3M_Rate_zscore_60d", "AMD_ret_5d", "DHR_vol_20d", "HUM_Humana_ret_5d", "HD_ret_1d", "CCI_CrownCastle_vol_20d", "EWA_Australia_ret_1d", "MS_MorganStanley_ret_5d", "XOM_ret_20d", "NVDA_vol_20d", "PLD_Prologis_ret_5d", "spx_vol_5d", "Nikkei_Japan_zscore_60d", "TED_Spread_vol_20d", "Retail_Sales_zscore_60d", "PPL_PPL_ret_1d", "EQR_Equity_ret_1d"], "is_new": true}, {"model_id": "new_h10_STRESS_XGBoost_N25_t4", "algo": "XGBoost", "regime": "STRESS", "horizon": 10, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "HD_zscore_60d", "TM_Telephone_vol_20d", "EWL_Switzerland_vol_20d", "LMT_LockheedMartin_vol_20d", "NEE_NextEra_ret_20d", "Core_CPI_zscore_60d", "CPB_CampbellSoup_ret_5d", "MS_MorganStanley_ret_1d", "XOM_ret_20d", "DAX_Germany_vol_20d", "PAYX_Paychex_zscore_60d", "EWL_Switzerland_zscore_60d", "EWM_Malaysia_zscore_60d", "SO_SouthernCo_ret_5d", "PLD_Prologis_ret_5d", "CMCSA_ret_1d", "US3M_Rate_vol_20d", "Core_PCE_zscore_60d", "SPY_zscore_60d", "EWQ_France_zscore_60d", "SLB_Schlumberger_ret_1d", "XLV_Health_zscore_60d", "T_ret_1d"], "is_new": true}, {"model_id": "new_h10_STRESS_XGBoost_N25_t5", "algo": "XGBoost", "regime": "STRESS", "horizon": 10, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "SLB_Schlumberger_ret_5d", "QQQ_vol_20d", "NEE_NextEra_ret_20d", "EWJ_Japan_vol_20d", "PAYX_Paychex_zscore_60d", "PLD_Prologis_ret_5d", "EWQ_France_zscore_60d", "BDX_Becton_Dickinson_ret_20d", "MS_MorganStanley_zscore_60d", "ASX_Australia_ret_5d", "EWA_Australia_zscore_60d", "heston_var_ev_h3", "Core_PCE_zscore_60d", "SBUX_ret_5d", "US3M_Rate_vol_20d", "DE_Deere_ret_5d", "SPY_zscore_60d", "NVDA_vol_20d", "AMD_ret_1d", "HangSeng_HK_vol_20d", "PAYX_Paychex_vol_20d", "hmm_p_stress", "EWQ_France_ret_20d"], "is_new": true}, {"model_id": "new_h10_STRESS_XGBoost_N25_t6", "algo": "XGBoost", "regime": "STRESS", "horizon": 10, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "TED_Spread_vol_20d", "gjr_condvar_h1", "SPY_zscore_60d", "EFFR_ret_1d", "AMD_ret_1d", "VRP_ma5", "LLY_zscore_60d", "BA_ret_1d", "XLY_Disc_vol_20d", "PLD_Prologis_ret_5d", "HangSeng_HK_vol_20d", "hmm_p_stress", "XLK_Tech_zscore_60d", "3M_vol_20d", "IYM_BasicMaterials_ret_20d", "SLB_Schlumberger_ret_1d", "AVB_AvalonBay_zscore_60d", "US3M_Rate_zscore_60d", "US1Y_Rate_ret_5d", "PPL_PPL_ret_1d", "BLK_BlackRock_zscore_60d", "PG_ret_20d", "BTI_BritishAmerican_ret_5d"], "is_new": true}, {"model_id": "new_h10_STRESS_XGBoost_N25_t7", "algo": "XGBoost", "regime": "STRESS", "horizon": 10, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EXC_Exelon_ret_1d", "LOW_Lowes_ret_20d", "EQR_Equity_ret_1d", "EXC_Exelon_zscore_60d", "INTC_ret_1d", "GILD_Gilead_ret_20d", "NEE_NextEra_ret_20d", "PLD_Prologis_ret_5d", "SBUX_vol_20d", "Industrial_Production_zscore_60d", "DOW_Price_zscore_60d", "CMCSA_ret_1d", "AMD_ret_5d", "TED_Spread_zscore_60d", "spx_vol_5d", "CPB_CampbellSoup_vol_20d", "VOD_Vodafone_zscore_60d", "XLK_Tech_zscore_60d", "HD_ret_20d", "EWJ_Japan_vol_20d", "TXN_vol_20d", "AMGN_Amgen_ret_1d", "EOG_EOGResources_vol_20d"], "is_new": true}, {"model_id": "new_h10_STRESS_XGBoost_N30_t0", "algo": "XGBoost", "regime": "STRESS", "horizon": 10, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "XLY_Disc_vol_20d", "CPB_CampbellSoup_ret_20d", "VRP_ma5", "CPB_CampbellSoup_zscore_60d", "NEE_NextEra_ret_20d", "AXP_Amex_vol_20d", "spx_vol_5d", "Core_CPI_zscore_60d", "DE_Deere_vol_20d", "MS_MorganStanley_ret_1d", "US1Y_Rate_ret_5d", "US5Y_Rate_ret_5d", "ORCL_zscore_60d", "Industrial_Production_zscore_60d", "EWQ_France_zscore_60d", "US6M_Rate_ret_20d", "IWM_SmallCap_vol_20d", "EWL_Switzerland_vol_20d", "DE_Deere_ret_5d", "EOG_EOGResources_ret_5d", "SBUX_ret_5d", "AMZN_ret_5d", "T_ret_1d", "Nikkei_Japan_zscore_60d", "LLY_zscore_60d", "SLB_Schlumberger_ret_1d", "VVIX_ret_20d", "PLD_Prologis_ret_5d"], "is_new": true}, {"model_id": "new_h10_STRESS_XGBoost_N30_t1", "algo": "XGBoost", "regime": "STRESS", "horizon": 10, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "SBUX_ret_5d", "Nikkei_Japan_zscore_60d", "US7Y_Rate_ret_20d", "LUV_SouthwestAir_ret_5d", "DAX_Germany_vol_20d", "VRP_ma5", "HangSeng_HK_ret_5d", "XLY_Disc_vol_20d", "spx_abs_ret_max_5d", "US3Y_Rate_ret_5d", "US3M_Rate_vol_20d", "XOM_ret_1d", "NVDA_vol_20d", "HUM_Humana_ret_5d", "T10Y2Y_Spread_ret_5d", "heston_var_ev_h3", "Core_PCE_zscore_60d", "INTC_ret_5d", "SCHW_Schwab_ret_5d", "SJM_JM_Smucker_ret_1d", "SO_SouthernCo_ret_5d", "CCI_CrownCastle_vol_20d", "EWY_Korea_zscore_60d", "MS_MorganStanley_zscore_60d", "MRK_Merck_zscore_60d", "US1Y_Rate_ret_5d", "PFE_ret_1d", "EWH_HongKong_ret_5d"], "is_new": true}, {"model_id": "new_h10_STRESS_XGBoost_N30_t2", "algo": "XGBoost", "regime": "STRESS", "horizon": 10, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "AMT_AmericanTower_ret_1d", "3M_vol_20d", "SBUX_ret_5d", "T10Y2Y_Spread_ret_5d", "EOG_EOGResources_vol_20d", "spx_momentum_3d", "Core_CPI_zscore_60d", "VOD_Vodafone_zscore_60d", "SJM_JM_Smucker_ret_1d", "IYR_US_REIT2_zscore_60d", "WTI_Oil_FRED_zscore_60d", "US7Y_Rate_ret_20d", "DE_Deere_ret_5d", "EWA_Australia_ret_1d", "AMD_ret_5d", "EWY_Korea_zscore_60d", "NFCI_ret_5d", "JNJ_ret_1d", "TGT_Target_zscore_60d", "SLB_Schlumberger_ret_1d", "Nikkei_Japan_zscore_60d", "EWL_Switzerland_vol_20d", "US5Y_Rate_ret_5d", "EWY_Korea_ret_20d", "CMCSA_ret_1d", "EQR_Equity_ret_1d", "HangSeng_HK_vol_20d", "TM_Telephone_vol_20d"], "is_new": true}, {"model_id": "new_h10_STRESS_XGBoost_N30_t3", "algo": "XGBoost", "regime": "STRESS", "horizon": 10, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "ASX_Australia_ret_5d", "M_Macys_vol_20d", "TED_Spread_vol_20d", "TGT_Target_zscore_60d", "BTI_BritishAmerican_ret_20d", "HUM_Humana_ret_5d", "PG_ret_20d", "WTI_Oil_FRED_zscore_60d", "SLB_Schlumberger_ret_1d", "HD_zscore_60d", "EWA_Australia_ret_1d", "CPB_CampbellSoup_zscore_60d", "EQIX_Equinix_ret_5d", "MS_MorganStanley_zscore_60d", "heston_ev_h3", "SBUX_ret_5d", "EWY_Korea_ret_20d", "US1Y_Rate_ret_5d", "ORCL_vol_20d", "T10Y2Y_Spread_ret_5d", "SO_SouthernCo_ret_5d", "3M_vol_20d", "US30Y_Rate_ret_20d", "CPB_CampbellSoup_vol_20d", "DAX_Germany_zscore_60d", "NEE_NextEra_ret_20d", "CLX_Clorox_vol_20d", "SPY_zscore_60d"], "is_new": true}, {"model_id": "new_h10_STRESS_XGBoost_N30_t4", "algo": "XGBoost", "regime": "STRESS", "horizon": 10, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "AXP_Amex_ret_20d", "ORCL_zscore_60d", "PAYX_Paychex_ret_20d", "HangSeng_HK_ret_1d", "SPY_zscore_60d", "LMT_LockheedMartin_ret_1d", "CPB_CampbellSoup_zscore_60d", "ASX_Australia_ret_5d", "Brent_Oil_FRED_ret_5d", "PAYX_Paychex_zscore_60d", "ITT_ITTInc_ret_5d", "heston_var_ev_h7", "ORCL_vol_20d", "PCAR_PaccarInc_ret_5d", "MSTR_Bitcoin3_ret_5d", "T_ret_1d", "EWY_Korea_ret_20d", "HUM_Humana_ret_5d", "DAX_Germany_zscore_60d", "MSTR_Bitcoin3_ret_20d", "3M_ret_5d", "EWS_Singapore_ret_5d", "LUV_SouthwestAir_ret_5d", "US3Y_Rate_ret_5d", "CI_Cigna_vol_20d", "heston_var_ev_h3", "BA_ret_1d", "CMCSA_ret_1d"], "is_new": true}, {"model_id": "new_h10_STRESS_XGBoost_N30_t5", "algo": "XGBoost", "regime": "STRESS", "horizon": 10, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "3M_vol_20d", "EWM_Malaysia_ret_1d", "SLB_Schlumberger_ret_5d", "SLB_Schlumberger_ret_1d", "DOW_Price_zscore_60d", "US3M_Rate_zscore_60d", "EFFR_vol_20d", "NFCI_ret_5d", "IYM_BasicMaterials_ret_20d", "PAYX_Paychex_ret_20d", "Core_PCE_zscore_60d", "M_Macys_vol_20d", "DIS_vol_20d", "Brent_Oil_FRED_ret_5d", "EWJ_Japan_vol_20d", "AMD_ret_1d", "EXC_Exelon_zscore_60d", "HD_ret_5d", "BDX_Becton_Dickinson_ret_20d", "HD_ret_20d", "SO_SouthernCo_ret_5d", "US1Y_Rate_ret_20d", "ORCL_zscore_60d", "HD_zscore_60d", "MS_MorganStanley_ret_5d", "Nikkei_Japan_zscore_60d", "CPB_CampbellSoup_ret_5d", "vix_mean_abs_ret_5d"], "is_new": true}, {"model_id": "new_h10_STRESS_XGBoost_N30_t6", "algo": "XGBoost", "regime": "STRESS", "horizon": 10, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWS_Singapore_ret_5d", "SBUX_zscore_60d", "NFCI_ret_5d", "PPL_PPL_ret_1d", "QQQ_vol_20d", "DE_Deere_vol_20d", "SJM_JM_Smucker_ret_1d", "ASX_Australia_ret_5d", "IWM_SmallCap_vol_20d", "PCAR_PaccarInc_ret_5d", "SPY_zscore_60d", "ORCL_vol_20d", "LLY_zscore_60d", "JNJ_ret_1d", "EFFR_ret_1d", "3M_vol_20d", "heston_var_ev_h3", "GD_GeneralDynamics_zscore_60d", "US1Y_Rate_ret_5d", "EWL_Switzerland_vol_20d", "PFE_ret_1d", "IBEX_Spain_ret_20d", "M_Macys_vol_20d", "AXP_Amex_ret_20d", "NWL_Newell_ret_20d", "HangSeng_HK_ret_5d", "heston_var_ev_h7", "spx_vol_5d"], "is_new": true}, {"model_id": "new_h10_STRESS_XGBoost_N30_t7", "algo": "XGBoost", "regime": "STRESS", "horizon": 10, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "heston_var_ev_h3", "HD_ret_5d", "FedFunds_zscore_60d", "AMD_ret_5d", "US1Y_Rate_ret_20d", "EWY_Korea_ret_20d", "EWA_Australia_ret_1d", "MS_MorganStanley_ret_1d", "US7Y_Rate_ret_20d", "DIS_vol_20d", "Brent_Oil_FRED_ret_5d", "IYR_US_REIT2_zscore_60d", "DHR_ret_1d", "NEE_NextEra_ret_20d", "AVB_AvalonBay_zscore_60d", "M_Macys_vol_20d", "HangSeng_HK_ret_1d", "HangSeng_HK_vol_20d", "CPB_CampbellSoup_ret_5d", "NWL_Newell_ret_20d", "XOM_ret_1d", "ES_Evergy_ret_1d", "AMD_ret_1d", "EWA_Australia_zscore_60d", "EXC_Exelon_zscore_60d", "MSTR_Bitcoin3_ret_1d", "TM_Telephone_ret_1d", "HD_ret_20d"], "is_new": true}, {"model_id": "new_h10_STRESS_LightGBM_N5_t0", "algo": "LightGBM", "regime": "STRESS", "horizon": 10, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "PAYX_Paychex_vol_20d", "Brent_Oil_FRED_ret_5d", "heston_var_ev_h7"], "is_new": true}, {"model_id": "new_h10_STRESS_LightGBM_N5_t1", "algo": "LightGBM", "regime": "STRESS", "horizon": 10, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "MO_AltriaMG_ret_1d", "PLD_Prologis_ret_5d", "EWQ_France_ret_20d"], "is_new": true}, {"model_id": "new_h10_STRESS_LightGBM_N5_t2", "algo": "LightGBM", "regime": "STRESS", "horizon": 10, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "FedFunds_zscore_60d", "US1Y_Rate_ret_5d", "DAX_Germany_vol_20d"], "is_new": true}, {"model_id": "new_h10_STRESS_LightGBM_N5_t3", "algo": "LightGBM", "regime": "STRESS", "horizon": 10, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "vix_mean_abs_ret_5d", "IWM_SmallCap_vol_20d", "PFE_ret_1d"], "is_new": true}, {"model_id": "new_h10_STRESS_LightGBM_N5_t4", "algo": "LightGBM", "regime": "STRESS", "horizon": 10, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "US3M_Rate_zscore_60d", "3M_vol_20d", "TED_Spread_vol_20d"], "is_new": true}, {"model_id": "new_h10_STRESS_LightGBM_N5_t5", "algo": "LightGBM", "regime": "STRESS", "horizon": 10, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "Core_PCE_zscore_60d", "T_ret_1d", "TM_Telephone_ret_1d"], "is_new": true}, {"model_id": "new_h10_STRESS_LightGBM_N5_t6", "algo": "LightGBM", "regime": "STRESS", "horizon": 10, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "TM_Telephone_vol_20d", "hmm_p_stress", "CPB_CampbellSoup_ret_20d"], "is_new": true}, {"model_id": "new_h10_STRESS_LightGBM_N5_t7", "algo": "LightGBM", "regime": "STRESS", "horizon": 10, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWM_Malaysia_zscore_60d", "EOG_EOGResources_ret_5d", "heston_var_ev_h3"], "is_new": true}, {"model_id": "new_h10_STRESS_LightGBM_N8_t0", "algo": "LightGBM", "regime": "STRESS", "horizon": 10, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "HangSeng_HK_ret_1d", "heston_var_ev_h3", "FedFunds_zscore_60d", "EXC_Exelon_ret_1d", "XLK_Tech_zscore_60d", "HUM_Humana_ret_5d"], "is_new": true}, {"model_id": "new_h10_STRESS_LightGBM_N8_t1", "algo": "LightGBM", "regime": "STRESS", "horizon": 10, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "HUM_Humana_ret_5d", "EQR_Equity_ret_1d", "IYM_BasicMaterials_ret_20d", "TED_Spread_zscore_60d", "SBUX_vol_20d", "XLK_Tech_zscore_60d"], "is_new": true}, {"model_id": "new_h10_STRESS_LightGBM_N8_t2", "algo": "LightGBM", "regime": "STRESS", "horizon": 10, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "US5Y_Rate_ret_5d", "PAYX_Paychex_ret_20d", "LMT_LockheedMartin_ret_1d", "CMCSA_ret_1d", "DHR_vol_20d", "EXC_Exelon_ret_1d"], "is_new": true}, {"model_id": "new_h10_STRESS_LightGBM_N8_t3", "algo": "LightGBM", "regime": "STRESS", "horizon": 10, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "HD_ret_5d", "IYR_US_REIT2_zscore_60d", "AVB_AvalonBay_zscore_60d", "VOD_Vodafone_zscore_60d", "DHR_ret_1d", "AXP_Amex_ret_20d"], "is_new": true}, {"model_id": "new_h10_STRESS_LightGBM_N8_t4", "algo": "LightGBM", "regime": "STRESS", "horizon": 10, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EOG_EOGResources_ret_5d", "DHR_ret_1d", "TM_Telephone_vol_20d", "JNJ_ret_1d", "3M_ret_5d", "DIS_vol_20d"], "is_new": true}, {"model_id": "new_h10_STRESS_LightGBM_N8_t5", "algo": "LightGBM", "regime": "STRESS", "horizon": 10, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "ENB_EnbridgeInc_ret_1d", "SCHW_Schwab_ret_5d", "MSTR_Bitcoin3_ret_1d", "EWS_Singapore_ret_5d", "MSTR_Bitcoin3_ret_5d", "MSTR_Bitcoin3_ret_20d"], "is_new": true}, {"model_id": "new_h10_STRESS_LightGBM_N8_t6", "algo": "LightGBM", "regime": "STRESS", "horizon": 10, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "HD_ret_5d", "LMT_LockheedMartin_vol_20d", "AVB_AvalonBay_zscore_60d", "TM_Telephone_vol_20d", "EWC_Canada_zscore_60d", "XLY_Disc_vol_20d"], "is_new": true}, {"model_id": "new_h10_STRESS_LightGBM_N8_t7", "algo": "LightGBM", "regime": "STRESS", "horizon": 10, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "PG_ret_20d", "NVDA_vol_20d", "US5Y_Rate_ret_5d", "LUV_SouthwestAir_ret_5d", "EWG_Germany_ret_20d", "M_Macys_vol_20d"], "is_new": true}, {"model_id": "new_h10_STRESS_LightGBM_N10_t0", "algo": "LightGBM", "regime": "STRESS", "horizon": 10, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "TED_Spread_zscore_60d", "US3Y_Rate_ret_5d", "SLB_Schlumberger_ret_1d", "PLD_Prologis_ret_5d", "T10Y2Y_Spread_ret_5d", "EWA_Australia_zscore_60d", "EQR_Equity_ret_1d", "MSTR_Bitcoin3_ret_5d"], "is_new": true}, {"model_id": "new_h10_STRESS_LightGBM_N10_t1", "algo": "LightGBM", "regime": "STRESS", "horizon": 10, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "Michigan_Sentiment_ret_20d", "PLD_Prologis_ret_5d", "LOW_Lowes_ret_20d", "heston_var_ev_h5", "SJM_JM_Smucker_ret_5d", "TED_Spread_vol_20d", "BTI_BritishAmerican_ret_20d", "EWJ_Japan_vol_20d"], "is_new": true}, {"model_id": "new_h10_STRESS_LightGBM_N10_t2", "algo": "LightGBM", "regime": "STRESS", "horizon": 10, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "SBUX_ret_5d", "NOC_Northrop_ret_20d", "LOW_Lowes_ret_20d", "VVIX_ret_20d", "INTC_ret_5d", "US3M_Rate_vol_20d", "BDX_Becton_Dickinson_ret_20d", "VRP_ma5"], "is_new": true}, {"model_id": "new_h10_STRESS_LightGBM_N10_t3", "algo": "LightGBM", "regime": "STRESS", "horizon": 10, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "AMD_ret_1d", "HUM_Humana_ret_5d", "QQQ_vol_20d", "BLK_BlackRock_zscore_60d", "EWL_Switzerland_zscore_60d", "HD_ret_20d", "3M_ret_5d", "IYR_US_REIT2_zscore_60d"], "is_new": true}, {"model_id": "new_h10_STRESS_LightGBM_N10_t4", "algo": "LightGBM", "regime": "STRESS", "horizon": 10, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "HD_ret_20d", "IYR_US_REIT2_zscore_60d", "TM_Telephone_ret_1d", "XLY_Disc_vol_20d", "EOG_EOGResources_vol_20d", "XOM_ret_1d", "PAYX_Paychex_zscore_60d", "AORD_AUS_zscore_60d"], "is_new": true}, {"model_id": "new_h10_STRESS_LightGBM_N10_t5", "algo": "LightGBM", "regime": "STRESS", "horizon": 10, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "US30Y_Rate_ret_20d", "ASX_Australia_ret_5d", "SPY_zscore_60d", "Michigan_Sentiment_ret_20d", "EWJ_Japan_vol_20d", "MS_MorganStanley_ret_5d", "Nikkei_Japan_vol_20d", "EFFR_vol_20d"], "is_new": true}, {"model_id": "new_h10_STRESS_LightGBM_N10_t6", "algo": "LightGBM", "regime": "STRESS", "horizon": 10, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWM_Malaysia_vol_20d", "AVB_AvalonBay_zscore_60d", "PCAR_PaccarInc_ret_5d", "Brent_Oil_FRED_ret_5d", "NEE_NextEra_ret_20d", "VVIX_ret_20d", "LOW_Lowes_ret_20d", "EMR_Emerson_ret_20d"], "is_new": true}, {"model_id": "new_h10_STRESS_LightGBM_N10_t7", "algo": "LightGBM", "regime": "STRESS", "horizon": 10, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "LMT_LockheedMartin_ret_1d", "MSTR_Bitcoin3_ret_20d", "US6M_Rate_ret_20d", "TGT_Target_zscore_60d", "PPL_PPL_ret_1d", "EWM_Malaysia_vol_20d", "IYM_BasicMaterials_ret_20d", "vix_acceleration_1d"], "is_new": true}, {"model_id": "new_h10_STRESS_LightGBM_N12_t0", "algo": "LightGBM", "regime": "STRESS", "horizon": 10, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "BTI_BritishAmerican_ret_5d", "HangSeng_HK_ret_5d", "SLB_Schlumberger_ret_1d", "XLY_Disc_vol_20d", "heston_ev_h3", "SBUX_vol_20d", "LOW_Lowes_ret_5d", "HUM_Humana_ret_5d", "SJM_JM_Smucker_ret_5d", "HD_ret_1d"], "is_new": true}, {"model_id": "new_h10_STRESS_LightGBM_N12_t1", "algo": "LightGBM", "regime": "STRESS", "horizon": 10, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "IYR_US_REIT2_zscore_60d", "ORCL_vol_20d", "EWM_Malaysia_vol_20d", "EWQ_France_ret_20d", "SO_SouthernCo_ret_5d", "PAYX_Paychex_ret_20d", "VOD_Vodafone_zscore_60d", "EWL_Switzerland_vol_20d", "EQR_Equity_ret_1d", "EWM_Malaysia_ret_1d"], "is_new": true}, {"model_id": "new_h10_STRESS_LightGBM_N12_t2", "algo": "LightGBM", "regime": "STRESS", "horizon": 10, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "PCAR_PaccarInc_ret_5d", "CPB_CampbellSoup_vol_20d", "ASX_Australia_vol_20d", "SJM_JM_Smucker_ret_1d", "DOW_Price_zscore_60d", "XOM_ret_20d", "NEE_NextEra_ret_20d", "EWY_Korea_zscore_60d", "MRK_Merck_zscore_60d", "MSTR_Bitcoin3_ret_1d"], "is_new": true}, {"model_id": "new_h10_STRESS_LightGBM_N12_t3", "algo": "LightGBM", "regime": "STRESS", "horizon": 10, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "DHR_ret_1d", "CCI_CrownCastle_vol_20d", "BDX_Becton_Dickinson_ret_20d", "MS_MorganStanley_ret_5d", "XLY_Disc_vol_20d", "HUM_Humana_ret_5d", "EWG_Germany_vol_20d", "PG_ret_20d", "XLK_Tech_zscore_60d", "MSTR_Bitcoin3_ret_20d"], "is_new": true}, {"model_id": "new_h10_STRESS_LightGBM_N12_t4", "algo": "LightGBM", "regime": "STRESS", "horizon": 10, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "NEE_NextEra_ret_20d", "GE_ret_1d", "T10Y2Y_Spread_ret_5d", "SBUX_zscore_60d", "SPY_zscore_60d", "DHR_ret_1d", "PFE_ret_1d", "spx_abs_ret_max_5d", "SO_SouthernCo_ret_5d", "PAYX_Paychex_vol_20d"], "is_new": true}, {"model_id": "new_h10_STRESS_LightGBM_N12_t5", "algo": "LightGBM", "regime": "STRESS", "horizon": 10, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "US1Y_Rate_ret_20d", "EQR_Equity_ret_1d", "EWH_HongKong_ret_5d", "MSTR_Bitcoin3_ret_1d", "EWA_Australia_ret_1d", "ASX_Australia_ret_5d", "NOC_Northrop_ret_20d", "DE_Deere_ret_5d", "PAYX_Paychex_zscore_60d", "PCAR_PaccarInc_ret_5d"], "is_new": true}, {"model_id": "new_h10_STRESS_LightGBM_N12_t6", "algo": "LightGBM", "regime": "STRESS", "horizon": 10, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EFFR_ret_1d", "VVIX_ret_20d", "TM_Telephone_ret_1d", "INTC_ret_1d", "MS_MorganStanley_ret_1d", "DHR_ret_1d", "EWQ_France_zscore_60d", "CPB_CampbellSoup_ret_5d", "CCI_CrownCastle_vol_20d", "MSTR_Bitcoin3_ret_1d"], "is_new": true}, {"model_id": "new_h10_STRESS_LightGBM_N12_t7", "algo": "LightGBM", "regime": "STRESS", "horizon": 10, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "DAX_Germany_zscore_60d", "DE_Deere_ret_5d", "NEE_NextEra_ret_20d", "SBUX_vol_20d", "US3M_Rate_vol_20d", "Michigan_Sentiment_ret_20d", "XOM_ret_20d", "hmm_p_stress", "VOD_Vodafone_zscore_60d", "US3M_Rate_zscore_60d"], "is_new": true}, {"model_id": "new_h10_STRESS_LightGBM_N15_t0", "algo": "LightGBM", "regime": "STRESS", "horizon": 10, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "FedFunds_zscore_60d", "Michigan_Sentiment_ret_20d", "XLB_Materials_zscore_60d", "HD_zscore_60d", "TM_Telephone_ret_1d", "SBUX_vol_20d", "CPB_CampbellSoup_ret_20d", "CLX_Clorox_vol_20d", "EQIX_Equinix_ret_5d", "ITT_ITTInc_ret_5d", "M_Macys_vol_20d", "PAYX_Paychex_vol_20d", "EWQ_France_ret_20d"], "is_new": true}, {"model_id": "new_h10_STRESS_LightGBM_N15_t1", "algo": "LightGBM", "regime": "STRESS", "horizon": 10, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "SBUX_vol_20d", "HangSeng_HK_ret_1d", "EWA_Australia_ret_1d", "LMT_LockheedMartin_ret_1d", "PCAR_PaccarInc_ret_5d", "ORCL_vol_20d", "JNJ_ret_1d", "XLV_Health_zscore_60d", "EWL_Switzerland_zscore_60d", "US3M_Rate_zscore_60d", "heston_ev_h3", "NFCI_ret_5d", "DOW_Price_zscore_60d"], "is_new": true}, {"model_id": "new_h10_STRESS_LightGBM_N15_t2", "algo": "LightGBM", "regime": "STRESS", "horizon": 10, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "FedFunds_zscore_60d", "US6M_Rate_ret_20d", "Brent_Oil_FRED_ret_5d", "MSTR_Bitcoin3_ret_20d", "CPB_CampbellSoup_vol_20d", "MSTR_Bitcoin3_ret_5d", "CMCSA_ret_1d", "ES_Evergy_ret_1d", "MS_MorganStanley_zscore_60d", "NOC_Northrop_ret_20d", "T10Y2Y_Spread_ret_5d", "CLX_Clorox_vol_20d", "SPY_zscore_60d"], "is_new": true}, {"model_id": "new_h10_STRESS_LightGBM_N15_t3", "algo": "LightGBM", "regime": "STRESS", "horizon": 10, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "LMT_LockheedMartin_vol_20d", "PG_ret_20d", "Michigan_Sentiment_ret_20d", "HangSeng_HK_ret_1d", "US7Y_Rate_ret_20d", "ENB_EnbridgeInc_ret_1d", "heston_var_ev_h5", "hmm_p_stress", "gjr_condvar_h1", "EQR_Equity_ret_1d", "XLF_Fin_vol_20d", "EOG_EOGResources_vol_20d", "AVB_AvalonBay_zscore_60d"], "is_new": true}, {"model_id": "new_h10_STRESS_LightGBM_N15_t4", "algo": "LightGBM", "regime": "STRESS", "horizon": 10, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "SJM_JM_Smucker_ret_1d", "EWC_Canada_zscore_60d", "LOW_Lowes_ret_5d", "XLB_Materials_zscore_60d", "spx_abs_ret_max_5d", "HangSeng_HK_ret_1d", "ASX_Australia_ret_5d", "INTC_ret_5d", "EWL_Switzerland_zscore_60d", "US3M_Rate_vol_20d", "US1Y_Rate_ret_20d", "spx_momentum_3d", "Nikkei_Japan_vol_20d"], "is_new": true}, {"model_id": "new_h10_STRESS_LightGBM_N15_t5", "algo": "LightGBM", "regime": "STRESS", "horizon": 10, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "AMD_ret_5d", "MS_MorganStanley_ret_5d", "CPB_CampbellSoup_ret_20d", "Brent_Oil_FRED_ret_20d", "Nikkei_Japan_zscore_60d", "EWQ_France_ret_20d", "US30Y_Rate_ret_20d", "US3Y_Rate_ret_5d", "HD_ret_5d", "GD_GeneralDynamics_zscore_60d", "US6M_Rate_ret_20d", "HangSeng_HK_vol_20d", "Core_CPI_zscore_60d"], "is_new": true}, {"model_id": "new_h10_STRESS_LightGBM_N15_t6", "algo": "LightGBM", "regime": "STRESS", "horizon": 10, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "XLB_Materials_zscore_60d", "EWG_Germany_vol_20d", "PG_ret_20d", "DE_Deere_vol_20d", "NWL_Newell_ret_20d", "CPB_CampbellSoup_ret_5d", "ES_Evergy_ret_1d", "heston_var_ev_h5", "US1Y_Rate_ret_5d", "vix_acceleration_1d", "TED_Spread_vol_20d", "DHR_ret_1d", "EMR_Emerson_ret_20d"], "is_new": true}, {"model_id": "new_h10_STRESS_LightGBM_N15_t7", "algo": "LightGBM", "regime": "STRESS", "horizon": 10, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "vix_acceleration_1d", "NFCI_ret_5d", "EMR_Emerson_ret_20d", "MS_MorganStanley_ret_5d", "BTI_BritishAmerican_ret_20d", "ES_Evergy_ret_1d", "ASX_Australia_ret_5d", "IWM_SmallCap_vol_20d", "ORCL_vol_20d", "US6M_Rate_ret_20d", "AXP_Amex_vol_20d", "PAYX_Paychex_ret_20d", "MS_MorganStanley_zscore_60d"], "is_new": true}, {"model_id": "new_h10_STRESS_LightGBM_N20_t0", "algo": "LightGBM", "regime": "STRESS", "horizon": 10, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "DE_Deere_vol_20d", "gjr_condvar_h1", "EFFR_vol_20d", "HangSeng_HK_vol_20d", "LLY_zscore_60d", "HangSeng_HK_ret_1d", "T10Y2Y_Spread_ret_5d", "IBEX_Spain_ret_20d", "MS_MorganStanley_zscore_60d", "NWL_Newell_ret_20d", "EWA_Australia_zscore_60d", "AMT_AmericanTower_ret_1d", "TM_Telephone_vol_20d", "SJM_JM_Smucker_ret_5d", "EWM_Malaysia_ret_1d", "ASX_Australia_vol_20d", "CMCSA_ret_1d", "EWC_Canada_zscore_60d"], "is_new": true}, {"model_id": "new_h10_STRESS_LightGBM_N20_t1", "algo": "LightGBM", "regime": "STRESS", "horizon": 10, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "vix_mean_abs_ret_5d", "EWA_Australia_ret_1d", "EWQ_France_ret_20d", "FedFunds_zscore_60d", "Industrial_Production_zscore_60d", "EWS_Singapore_ret_5d", "SBUX_vol_20d", "LUV_SouthwestAir_ret_5d", "AMD_ret_5d", "DOW_Price_zscore_60d", "CPB_CampbellSoup_ret_5d", "XOM_ret_1d", "3M_ret_5d", "DIS_vol_20d", "EWY_Korea_zscore_60d", "CPB_CampbellSoup_ret_20d", "EWM_Malaysia_vol_20d", "gjr_condvar_h1"], "is_new": true}, {"model_id": "new_h10_STRESS_LightGBM_N20_t2", "algo": "LightGBM", "regime": "STRESS", "horizon": 10, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "SBUX_ret_5d", "US1Y_Rate_ret_5d", "EWJ_Japan_vol_20d", "SBUX_vol_20d", "VRP_ma5", "EWY_Korea_ret_20d", "Retail_Sales_zscore_60d", "3M_ret_5d", "TGT_Target_zscore_60d", "MS_MorganStanley_zscore_60d", "EWC_Canada_zscore_60d", "SO_SouthernCo_ret_5d", "DHR_vol_20d", "US7Y_Rate_ret_20d", "TM_Telephone_vol_20d", "TXN_vol_20d", "LMT_LockheedMartin_ret_1d", "DE_Deere_vol_20d"], "is_new": true}, {"model_id": "new_h10_STRESS_LightGBM_N20_t3", "algo": "LightGBM", "regime": "STRESS", "horizon": 10, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "CLX_Clorox_vol_20d", "CCI_CrownCastle_vol_20d", "CPB_CampbellSoup_ret_20d", "EWG_Germany_ret_20d", "EOG_EOGResources_vol_20d", "3M_ret_5d", "T_ret_1d", "CPB_CampbellSoup_vol_20d", "ITT_ITTInc_ret_5d", "EWY_Korea_zscore_60d", "EFFR_ret_1d", "M_Macys_vol_20d", "US6M_Rate_ret_20d", "LMT_LockheedMartin_vol_20d", "AMT_AmericanTower_ret_1d", "AMD_ret_5d", "Brent_Oil_FRED_ret_20d", "XOM_ret_1d"], "is_new": true}, {"model_id": "new_h10_STRESS_LightGBM_N20_t4", "algo": "LightGBM", "regime": "STRESS", "horizon": 10, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "TM_Telephone_vol_20d", "Retail_Sales_zscore_60d", "EWM_Malaysia_zscore_60d", "PAYX_Paychex_ret_20d", "IYR_US_REIT2_zscore_60d", "Nikkei_Japan_vol_20d", "CLX_Clorox_vol_20d", "NVDA_vol_20d", "US1Y_Rate_ret_5d", "DAX_Germany_vol_20d", "IBEX_Spain_ret_20d", "EWA_Australia_zscore_60d", "heston_ev_h3", "PG_ret_20d", "spx_abs_ret_max_5d", "LMT_LockheedMartin_ret_1d", "EWY_Korea_zscore_60d", "GILD_Gilead_ret_20d"], "is_new": true}, {"model_id": "new_h10_STRESS_LightGBM_N20_t5", "algo": "LightGBM", "regime": "STRESS", "horizon": 10, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "MRK_Merck_zscore_60d", "heston_var_ev_h7", "DE_Deere_vol_20d", "Industrial_Production_zscore_60d", "FedFunds_zscore_60d", "EWL_Switzerland_zscore_60d", "EFFR_ret_1d", "MS_MorganStanley_ret_5d", "EWM_Malaysia_zscore_60d", "LUV_SouthwestAir_ret_5d", "BA_ret_1d", "XLK_Tech_zscore_60d", "EOG_EOGResources_vol_20d", "Nikkei_Japan_zscore_60d", "EWG_Germany_vol_20d", "Michigan_Sentiment_ret_20d", "VRP_ma5", "BLK_BlackRock_zscore_60d"], "is_new": true}, {"model_id": "new_h10_STRESS_LightGBM_N20_t6", "algo": "LightGBM", "regime": "STRESS", "horizon": 10, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "Retail_Sales_zscore_60d", "MSTR_Bitcoin3_ret_5d", "EWQ_France_ret_20d", "AMZN_ret_5d", "EWM_Malaysia_ret_1d", "EQIX_Equinix_ret_5d", "SCHW_Schwab_ret_5d", "Core_PCE_zscore_60d", "TED_Spread_vol_20d", "EWY_Korea_zscore_60d", "EWJ_Japan_vol_20d", "TXN_vol_20d", "EWS_Singapore_ret_5d", "DOW_Price_zscore_60d", "PCAR_PaccarInc_ret_5d", "LLY_zscore_60d", "AMD_ret_1d", "PAYX_Paychex_ret_20d"], "is_new": true}, {"model_id": "new_h10_STRESS_LightGBM_N20_t7", "algo": "LightGBM", "regime": "STRESS", "horizon": 10, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "HD_zscore_60d", "HD_ret_5d", "ENB_EnbridgeInc_ret_1d", "XOM_ret_1d", "PG_ret_20d", "Michigan_Sentiment_ret_20d", "3M_vol_20d", "EWJ_Japan_vol_20d", "EXC_Exelon_ret_1d", "NWL_Newell_ret_20d", "CPB_CampbellSoup_ret_5d", "spx_vol_5d", "Brent_Oil_FRED_ret_20d", "SJM_JM_Smucker_ret_1d", "EWS_Singapore_ret_5d", "TXN_vol_20d", "M_Macys_vol_20d", "EWG_Germany_vol_20d"], "is_new": true}, {"model_id": "new_h10_STRESS_LightGBM_N25_t0", "algo": "LightGBM", "regime": "STRESS", "horizon": 10, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWG_Germany_ret_20d", "CPB_CampbellSoup_ret_20d", "LMT_LockheedMartin_vol_20d", "INTC_ret_5d", "gjr_condvar_h1", "XLK_Tech_zscore_60d", "EWA_Australia_ret_1d", "AMZN_ret_5d", "HangSeng_HK_ret_1d", "Brent_Oil_FRED_ret_5d", "T10Y2Y_Spread_ret_5d", "MSTR_Bitcoin3_ret_5d", "CMCSA_ret_1d", "AORD_AUS_zscore_60d", "SBUX_ret_5d", "MSTR_Bitcoin3_ret_20d", "PG_ret_20d", "EQIX_Equinix_ret_5d", "SLB_Schlumberger_ret_5d", "BLK_BlackRock_zscore_60d", "IYR_US_REIT2_zscore_60d", "AXP_Amex_vol_20d", "HD_ret_1d"], "is_new": true}, {"model_id": "new_h10_STRESS_LightGBM_N25_t1", "algo": "LightGBM", "regime": "STRESS", "horizon": 10, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "BTI_BritishAmerican_ret_5d", "DIS_vol_20d", "VRP_ma5", "ASX_Australia_ret_5d", "MSTR_Bitcoin3_ret_1d", "EWQ_France_ret_20d", "DAX_Germany_vol_20d", "GD_GeneralDynamics_zscore_60d", "PFE_ret_1d", "US3M_Rate_vol_20d", "vix_mean_abs_ret_5d", "IBEX_Spain_ret_20d", "EWM_Malaysia_vol_20d", "US6M_Rate_ret_20d", "FedFunds_zscore_60d", "DHR_vol_20d", "IYR_US_REIT2_zscore_60d", "SBUX_ret_5d", "SJM_JM_Smucker_ret_1d", "LMT_LockheedMartin_ret_1d", "CLX_Clorox_vol_20d", "EWL_Switzerland_vol_20d", "T10Y2Y_Spread_ret_5d"], "is_new": true}, {"model_id": "new_h10_STRESS_LightGBM_N25_t2", "algo": "LightGBM", "regime": "STRESS", "horizon": 10, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "HangSeng_HK_ret_5d", "EMR_Emerson_ret_20d", "XOM_ret_1d", "EWM_Malaysia_zscore_60d", "EWY_Korea_zscore_60d", "IWM_SmallCap_vol_20d", "FedFunds_zscore_60d", "NEE_NextEra_ret_20d", "vix_acceleration_1d", "NOC_Northrop_ret_20d", "heston_var_ev_h5", "MRK_Merck_zscore_60d", "PPL_PPL_ret_1d", "INTC_ret_5d", "HD_ret_20d", "AMD_ret_1d", "CMCSA_ret_1d", "PG_ret_20d", "ENB_EnbridgeInc_ret_1d", "AXP_Amex_ret_20d", "IBEX_Spain_ret_20d", "XLV_Health_zscore_60d", "3M_vol_20d"], "is_new": true}, {"model_id": "new_h10_STRESS_LightGBM_N25_t3", "algo": "LightGBM", "regime": "STRESS", "horizon": 10, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "ITT_ITTInc_ret_5d", "LMT_LockheedMartin_vol_20d", "XLB_Materials_zscore_60d", "HD_ret_20d", "DHR_ret_1d", "CTAS_Cintas_vol_20d", "IBEX_Spain_ret_20d", "CPB_CampbellSoup_zscore_60d", "AMD_ret_1d", "NOC_Northrop_ret_20d", "PPL_PPL_ret_1d", "US6M_Rate_ret_20d", "EXC_Exelon_ret_1d", "EWM_Malaysia_ret_1d", "EWA_Australia_zscore_60d", "DE_Deere_vol_20d", "BDX_Becton_Dickinson_ret_20d", "EWL_Switzerland_zscore_60d", "DAX_Germany_vol_20d", "heston_var_ev_h3", "AMGN_Amgen_ret_1d", "MS_MorganStanley_ret_5d", "IWM_SmallCap_vol_20d"], "is_new": true}, {"model_id": "new_h10_STRESS_LightGBM_N25_t4", "algo": "LightGBM", "regime": "STRESS", "horizon": 10, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "CPB_CampbellSoup_ret_20d", "EWM_Malaysia_vol_20d", "EMR_Emerson_ret_20d", "EWY_Korea_ret_20d", "AMD_ret_5d", "HD_ret_20d", "ENB_EnbridgeInc_ret_1d", "HangSeng_HK_vol_20d", "heston_var_ev_h5", "Brent_Oil_FRED_ret_20d", "HD_ret_1d", "US3Y_Rate_ret_5d", "CPB_CampbellSoup_vol_20d", "PAYX_Paychex_zscore_60d", "CTAS_Cintas_vol_20d", "SCHW_Schwab_ret_5d", "US6M_Rate_ret_20d", "ES_Evergy_ret_1d", "EWG_Germany_ret_20d", "EWQ_France_zscore_60d", "EWM_Malaysia_ret_1d", "VOD_Vodafone_zscore_60d", "DE_Deere_ret_5d"], "is_new": true}, {"model_id": "new_h10_STRESS_LightGBM_N25_t5", "algo": "LightGBM", "regime": "STRESS", "horizon": 10, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "T_ret_1d", "CPB_CampbellSoup_vol_20d", "MS_MorganStanley_ret_5d", "CPB_CampbellSoup_ret_20d", "VOD_Vodafone_zscore_60d", "ENB_EnbridgeInc_ret_1d", "AMGN_Amgen_ret_1d", "EWA_Australia_zscore_60d", "NVDA_vol_20d", "LOW_Lowes_ret_5d", "US1Y_Rate_ret_5d", "GILD_Gilead_ret_20d", "Core_CPI_zscore_60d", "BDX_Becton_Dickinson_ret_20d", "PFE_ret_1d", "LOW_Lowes_ret_20d", "DE_Deere_ret_5d", "T10Y2Y_Spread_ret_5d", "CI_Cigna_vol_20d", "BTI_BritishAmerican_ret_20d", "Nikkei_Japan_zscore_60d", "DIS_vol_20d", "Michigan_Sentiment_ret_20d"], "is_new": true}, {"model_id": "new_h10_STRESS_LightGBM_N25_t6", "algo": "LightGBM", "regime": "STRESS", "horizon": 10, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "vix_mean_abs_ret_5d", "EWL_Switzerland_zscore_60d", "DHR_ret_1d", "ENB_EnbridgeInc_ret_1d", "EWA_Australia_zscore_60d", "EWG_Germany_ret_20d", "CCI_CrownCastle_vol_20d", "Nikkei_Japan_vol_20d", "SJM_JM_Smucker_ret_5d", "TXN_vol_20d", "T10Y2Y_Spread_ret_5d", "LUV_SouthwestAir_ret_5d", "Core_PCE_zscore_60d", "XOM_ret_1d", "M_Macys_vol_20d", "hmm_p_stress", "DIS_vol_20d", "PLD_Prologis_ret_5d", "CPB_CampbellSoup_ret_5d", "XOM_ret_20d", "HangSeng_HK_ret_5d", "DE_Deere_vol_20d", "Retail_Sales_zscore_60d"], "is_new": true}, {"model_id": "new_h10_STRESS_LightGBM_N25_t7", "algo": "LightGBM", "regime": "STRESS", "horizon": 10, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "AVB_AvalonBay_zscore_60d", "EWJ_Japan_vol_20d", "LOW_Lowes_ret_20d", "AORD_AUS_zscore_60d", "BDX_Becton_Dickinson_ret_20d", "SO_SouthernCo_ret_5d", "M_Macys_vol_20d", "US5Y_Rate_ret_5d", "QQQ_vol_20d", "HangSeng_HK_vol_20d", "HangSeng_HK_ret_1d", "ORCL_vol_20d", "DE_Deere_vol_20d", "MO_AltriaMG_ret_1d", "XOM_ret_20d", "IWM_SmallCap_vol_20d", "EWA_Australia_ret_1d", "XOM_ret_1d", "ASX_Australia_ret_5d", "EWL_Switzerland_vol_20d", "SPY_zscore_60d", "HD_ret_1d", "VVIX_ret_20d"], "is_new": true}, {"model_id": "new_h10_STRESS_LightGBM_N30_t0", "algo": "LightGBM", "regime": "STRESS", "horizon": 10, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "US3M_Rate_vol_20d", "ORCL_zscore_60d", "GE_ret_1d", "EWQ_France_ret_20d", "HD_ret_20d", "HangSeng_HK_ret_5d", "CLX_Clorox_vol_20d", "PFE_ret_1d", "TXN_vol_20d", "SLB_Schlumberger_ret_1d", "AMD_ret_5d", "AXP_Amex_vol_20d", "SBUX_vol_20d", "BTI_BritishAmerican_ret_5d", "XLF_Fin_vol_20d", "ES_Evergy_ret_1d", "XLY_Disc_vol_20d", "AMT_AmericanTower_ret_1d", "PAYX_Paychex_zscore_60d", "CPB_CampbellSoup_vol_20d", "EWY_Korea_zscore_60d", "Brent_Oil_FRED_ret_5d", "heston_var_ev_h3", "DOW_Price_zscore_60d", "EWG_Germany_vol_20d", "TGT_Target_zscore_60d", "NOC_Northrop_ret_20d", "AVB_AvalonBay_zscore_60d"], "is_new": true}, {"model_id": "new_h10_STRESS_LightGBM_N30_t1", "algo": "LightGBM", "regime": "STRESS", "horizon": 10, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "BTI_BritishAmerican_ret_5d", "US3Y_Rate_ret_5d", "BA_ret_1d", "AMT_AmericanTower_ret_1d", "NOC_Northrop_ret_20d", "XLF_Fin_vol_20d", "ITT_ITTInc_ret_5d", "HangSeng_HK_ret_5d", "SCHW_Schwab_ret_5d", "EWQ_France_ret_20d", "VVIX_ret_20d", "HangSeng_HK_vol_20d", "BLK_BlackRock_zscore_60d", "MSTR_Bitcoin3_ret_20d", "CLX_Clorox_vol_20d", "ORCL_vol_20d", "heston_var_ev_h5", "T10Y2Y_Spread_ret_5d", "CTAS_Cintas_vol_20d", "SO_SouthernCo_ret_5d", "SLB_Schlumberger_ret_5d", "US7Y_Rate_ret_20d", "SJM_JM_Smucker_ret_5d", "PFE_ret_1d", "GILD_Gilead_ret_20d", "EWM_Malaysia_ret_1d", "CPB_CampbellSoup_vol_20d", "INTC_ret_1d"], "is_new": true}, {"model_id": "new_h10_STRESS_LightGBM_N30_t2", "algo": "LightGBM", "regime": "STRESS", "horizon": 10, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "US1Y_Rate_ret_20d", "LOW_Lowes_ret_20d", "DHR_ret_1d", "CI_Cigna_vol_20d", "EWC_Canada_zscore_60d", "EQIX_Equinix_ret_5d", "WTI_Oil_FRED_zscore_60d", "BTI_BritishAmerican_ret_5d", "ASX_Australia_vol_20d", "Michigan_Sentiment_ret_20d", "M_Macys_vol_20d", "XLV_Health_zscore_60d", "VVIX_ret_20d", "HangSeng_HK_ret_5d", "QQQ_vol_20d", "IYR_US_REIT2_zscore_60d", "EWS_Singapore_ret_5d", "PAYX_Paychex_zscore_60d", "PPL_PPL_ret_1d", "heston_var_ev_h7", "LLY_zscore_60d", "PCAR_PaccarInc_ret_5d", "EWA_Australia_ret_1d", "BDX_Becton_Dickinson_ret_20d", "LOW_Lowes_ret_5d", "MS_MorganStanley_ret_1d", "DHR_vol_20d", "Brent_Oil_FRED_ret_5d"], "is_new": true}, {"model_id": "new_h10_STRESS_LightGBM_N30_t3", "algo": "LightGBM", "regime": "STRESS", "horizon": 10, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "Retail_Sales_zscore_60d", "CPB_CampbellSoup_vol_20d", "Nikkei_Japan_vol_20d", "SBUX_vol_20d", "ENB_EnbridgeInc_ret_1d", "NEE_NextEra_ret_20d", "EWM_Malaysia_vol_20d", "EWQ_France_zscore_60d", "DHR_ret_1d", "HangSeng_HK_ret_1d", "TM_Telephone_vol_20d", "EMR_Emerson_ret_20d", "spx_momentum_3d", "DE_Deere_ret_5d", "Core_PCE_zscore_60d", "ASX_Australia_ret_5d", "IYR_US_REIT2_zscore_60d", "T10Y2Y_Spread_ret_5d", "MS_MorganStanley_ret_5d", "gjr_condvar_h1", "EWA_Australia_zscore_60d", "AXP_Amex_vol_20d", "SJM_JM_Smucker_ret_5d", "DAX_Germany_zscore_60d", "MS_MorganStanley_zscore_60d", "ASX_Australia_vol_20d", "CPB_CampbellSoup_ret_5d", "LLY_zscore_60d"], "is_new": true}, {"model_id": "new_h10_STRESS_LightGBM_N30_t4", "algo": "LightGBM", "regime": "STRESS", "horizon": 10, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "LOW_Lowes_ret_5d", "ITT_ITTInc_ret_5d", "EWJ_Japan_vol_20d", "SLB_Schlumberger_ret_5d", "INTC_ret_1d", "EWH_HongKong_ret_5d", "CTAS_Cintas_vol_20d", "XOM_ret_20d", "XLV_Health_zscore_60d", "EWA_Australia_zscore_60d", "HangSeng_HK_ret_5d", "spx_momentum_3d", "MSTR_Bitcoin3_ret_1d", "HangSeng_HK_vol_20d", "SBUX_zscore_60d", "XOM_ret_1d", "Michigan_Sentiment_ret_20d", "EWS_Singapore_ret_5d", "GILD_Gilead_ret_20d", "DAX_Germany_vol_20d", "PAYX_Paychex_ret_20d", "ES_Evergy_ret_1d", "CPB_CampbellSoup_zscore_60d", "Brent_Oil_FRED_ret_20d", "IYR_US_REIT2_zscore_60d", "TGT_Target_zscore_60d", "VRP_ma5", "EQR_Equity_ret_1d"], "is_new": true}, {"model_id": "new_h10_STRESS_LightGBM_N30_t5", "algo": "LightGBM", "regime": "STRESS", "horizon": 10, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EMR_Emerson_ret_20d", "MS_MorganStanley_zscore_60d", "EWY_Korea_zscore_60d", "DAX_Germany_vol_20d", "WTI_Oil_FRED_zscore_60d", "EXC_Exelon_zscore_60d", "Core_PCE_zscore_60d", "US6M_Rate_ret_20d", "PFE_ret_1d", "HangSeng_HK_vol_20d", "EWL_Switzerland_vol_20d", "ORCL_vol_20d", "EOG_EOGResources_vol_20d", "PAYX_Paychex_zscore_60d", "CI_Cigna_vol_20d", "DOW_Price_zscore_60d", "Brent_Oil_FRED_ret_20d", "EWQ_France_ret_20d", "QQQ_vol_20d", "CPB_CampbellSoup_vol_20d", "HangSeng_HK_ret_1d", "MRK_Merck_zscore_60d", "CLX_Clorox_vol_20d", "EWG_Germany_ret_20d", "DHR_vol_20d", "ES_Evergy_ret_1d", "heston_var_ev_h3", "SJM_JM_Smucker_ret_5d"], "is_new": true}, {"model_id": "new_h10_STRESS_LightGBM_N30_t6", "algo": "LightGBM", "regime": "STRESS", "horizon": 10, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "Michigan_Sentiment_ret_20d", "hmm_p_stress", "BTI_BritishAmerican_ret_20d", "EOG_EOGResources_ret_5d", "BTI_BritishAmerican_ret_5d", "IYR_US_REIT2_zscore_60d", "NFCI_ret_5d", "VOD_Vodafone_zscore_60d", "CPB_CampbellSoup_zscore_60d", "AXP_Amex_ret_20d", "XOM_ret_20d", "US1Y_Rate_ret_5d", "vix_acceleration_1d", "CPB_CampbellSoup_ret_5d", "CI_Cigna_vol_20d", "PAYX_Paychex_ret_20d", "NWL_Newell_ret_20d", "heston_var_ev_h7", "HD_ret_1d", "M_Macys_vol_20d", "Retail_Sales_zscore_60d", "Core_PCE_zscore_60d", "XLV_Health_zscore_60d", "US5Y_Rate_ret_5d", "NOC_Northrop_ret_20d", "Brent_Oil_FRED_ret_5d", "HUM_Humana_ret_5d", "ASX_Australia_vol_20d"], "is_new": true}, {"model_id": "new_h10_STRESS_LightGBM_N30_t7", "algo": "LightGBM", "regime": "STRESS", "horizon": 10, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "MSTR_Bitcoin3_ret_20d", "SLB_Schlumberger_ret_1d", "CMCSA_ret_1d", "Industrial_Production_zscore_60d", "PAYX_Paychex_ret_20d", "heston_var_ev_h3", "HangSeng_HK_ret_1d", "EWM_Malaysia_vol_20d", "MS_MorganStanley_ret_5d", "PCAR_PaccarInc_ret_5d", "CTAS_Cintas_vol_20d", "IYM_BasicMaterials_ret_20d", "DE_Deere_vol_20d", "SPY_zscore_60d", "EWG_Germany_vol_20d", "DOW_Price_zscore_60d", "Nikkei_Japan_vol_20d", "HD_ret_20d", "EFFR_vol_20d", "Nikkei_Japan_zscore_60d", "EFFR_ret_1d", "EQR_Equity_ret_1d", "vix_mean_abs_ret_5d", "NOC_Northrop_ret_20d", "heston_var_ev_h7", "3M_vol_20d", "ASX_Australia_ret_5d", "BTI_BritishAmerican_ret_5d"], "is_new": true}, {"model_id": "new_h10_STRESS_GradientBoosting_N5_t0", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 10, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EOG_EOGResources_vol_20d", "LMT_LockheedMartin_ret_1d", "Nikkei_Japan_zscore_60d"], "is_new": true}, {"model_id": "new_h10_STRESS_GradientBoosting_N5_t1", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 10, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "CMCSA_ret_1d", "VOD_Vodafone_zscore_60d", "AMGN_Amgen_ret_1d"], "is_new": true}, {"model_id": "new_h10_STRESS_GradientBoosting_N5_t2", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 10, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "WTI_Oil_FRED_zscore_60d", "MSTR_Bitcoin3_ret_20d", "SBUX_zscore_60d"], "is_new": true}, {"model_id": "new_h10_STRESS_GradientBoosting_N5_t3", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 10, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "Michigan_Sentiment_ret_20d", "SJM_JM_Smucker_ret_1d", "CPB_CampbellSoup_zscore_60d"], "is_new": true}, {"model_id": "new_h10_STRESS_GradientBoosting_N5_t4", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 10, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "ORCL_vol_20d", "SO_SouthernCo_ret_5d", "HangSeng_HK_vol_20d"], "is_new": true}, {"model_id": "new_h10_STRESS_GradientBoosting_N5_t5", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 10, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "DAX_Germany_vol_20d", "DE_Deere_ret_5d", "LLY_zscore_60d"], "is_new": true}, {"model_id": "new_h10_STRESS_GradientBoosting_N5_t6", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 10, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "SBUX_ret_5d", "vix_mean_abs_ret_5d", "CCI_CrownCastle_vol_20d"], "is_new": true}, {"model_id": "new_h10_STRESS_GradientBoosting_N5_t7", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 10, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "heston_ev_h3", "MS_MorganStanley_ret_5d", "EWS_Singapore_ret_5d"], "is_new": true}, {"model_id": "new_h10_STRESS_GradientBoosting_N8_t0", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 10, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "ES_Evergy_ret_1d", "MSTR_Bitcoin3_ret_5d", "LOW_Lowes_ret_5d", "ASX_Australia_ret_5d", "CMCSA_ret_1d", "T_ret_1d"], "is_new": true}, {"model_id": "new_h10_STRESS_GradientBoosting_N8_t1", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 10, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "US30Y_Rate_ret_20d", "WTI_Oil_FRED_zscore_60d", "ITT_ITTInc_ret_5d", "XOM_ret_1d", "NOC_Northrop_ret_20d", "EWL_Switzerland_zscore_60d"], "is_new": true}, {"model_id": "new_h10_STRESS_GradientBoosting_N8_t2", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 10, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "TED_Spread_vol_20d", "PAYX_Paychex_zscore_60d", "heston_var_ev_h3", "EQR_Equity_ret_1d", "CPB_CampbellSoup_ret_20d", "HD_ret_20d"], "is_new": true}, {"model_id": "new_h10_STRESS_GradientBoosting_N8_t3", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 10, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "SLB_Schlumberger_ret_1d", "DE_Deere_vol_20d", "SPY_zscore_60d", "NEE_NextEra_ret_20d", "EWJ_Japan_vol_20d", "US7Y_Rate_ret_20d"], "is_new": true}, {"model_id": "new_h10_STRESS_GradientBoosting_N8_t4", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 10, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "NEE_NextEra_ret_20d", "vix_acceleration_1d", "Nikkei_Japan_vol_20d", "EWY_Korea_zscore_60d", "PG_ret_20d", "GD_GeneralDynamics_zscore_60d"], "is_new": true}, {"model_id": "new_h10_STRESS_GradientBoosting_N8_t5", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 10, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "DIS_vol_20d", "SBUX_zscore_60d", "hmm_p_stress", "VOD_Vodafone_zscore_60d", "PG_ret_20d", "NWL_Newell_ret_20d"], "is_new": true}, {"model_id": "new_h10_STRESS_GradientBoosting_N8_t6", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 10, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "SBUX_vol_20d", "QQQ_vol_20d", "AVB_AvalonBay_zscore_60d", "BDX_Becton_Dickinson_ret_20d", "VVIX_ret_20d", "heston_var_ev_h7"], "is_new": true}, {"model_id": "new_h10_STRESS_GradientBoosting_N8_t7", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 10, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "GILD_Gilead_ret_20d", "ORCL_vol_20d", "INTC_ret_5d", "EOG_EOGResources_ret_5d", "EWS_Singapore_ret_5d", "EFFR_vol_20d"], "is_new": true}, {"model_id": "new_h10_STRESS_GradientBoosting_N10_t0", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 10, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "ORCL_vol_20d", "Core_PCE_zscore_60d", "EWM_Malaysia_ret_1d", "EWL_Switzerland_zscore_60d", "EWC_Canada_zscore_60d", "EWQ_France_ret_20d", "MSTR_Bitcoin3_ret_5d", "MO_AltriaMG_ret_1d"], "is_new": true}, {"model_id": "new_h10_STRESS_GradientBoosting_N10_t1", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 10, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "GILD_Gilead_ret_20d", "AORD_AUS_zscore_60d", "spx_vol_5d", "JNJ_ret_1d", "TM_Telephone_vol_20d", "Retail_Sales_zscore_60d", "CPB_CampbellSoup_vol_20d", "spx_abs_ret_max_5d"], "is_new": true}, {"model_id": "new_h10_STRESS_GradientBoosting_N10_t2", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 10, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "HangSeng_HK_ret_5d", "Brent_Oil_FRED_ret_5d", "SO_SouthernCo_ret_5d", "NWL_Newell_ret_20d", "TM_Telephone_vol_20d", "ORCL_vol_20d", "PAYX_Paychex_vol_20d", "GE_ret_1d"], "is_new": true}, {"model_id": "new_h10_STRESS_GradientBoosting_N10_t3", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 10, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "INTC_ret_1d", "MS_MorganStanley_zscore_60d", "XOM_ret_20d", "HD_zscore_60d", "BTI_BritishAmerican_ret_20d", "Nikkei_Japan_zscore_60d", "spx_momentum_3d", "BDX_Becton_Dickinson_ret_20d"], "is_new": true}, {"model_id": "new_h10_STRESS_GradientBoosting_N10_t4", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 10, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWM_Malaysia_ret_1d", "QQQ_vol_20d", "EXC_Exelon_zscore_60d", "SO_SouthernCo_ret_5d", "EWA_Australia_ret_1d", "EQIX_Equinix_ret_5d", "EFFR_ret_1d", "gjr_condvar_h1"], "is_new": true}, {"model_id": "new_h10_STRESS_GradientBoosting_N10_t5", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 10, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "SLB_Schlumberger_ret_5d", "heston_var_ev_h7", "3M_ret_5d", "3M_vol_20d", "MSTR_Bitcoin3_ret_20d", "XLY_Disc_vol_20d", "HangSeng_HK_ret_5d", "NFCI_ret_5d"], "is_new": true}, {"model_id": "new_h10_STRESS_GradientBoosting_N10_t6", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 10, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "HangSeng_HK_ret_1d", "AMD_ret_5d", "JNJ_ret_1d", "PCAR_PaccarInc_ret_5d", "CLX_Clorox_vol_20d", "US1Y_Rate_ret_5d", "TED_Spread_vol_20d", "NWL_Newell_ret_20d"], "is_new": true}, {"model_id": "new_h10_STRESS_GradientBoosting_N10_t7", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 10, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "SLB_Schlumberger_ret_5d", "XLB_Materials_zscore_60d", "Industrial_Production_zscore_60d", "CMCSA_ret_1d", "SBUX_zscore_60d", "ENB_EnbridgeInc_ret_1d", "ITT_ITTInc_ret_5d", "DE_Deere_vol_20d"], "is_new": true}, {"model_id": "new_h10_STRESS_GradientBoosting_N12_t0", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 10, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "MSTR_Bitcoin3_ret_5d", "Nikkei_Japan_vol_20d", "CPB_CampbellSoup_vol_20d", "VRP_ma5", "CPB_CampbellSoup_ret_5d", "EQIX_Equinix_ret_5d", "DHR_vol_20d", "SLB_Schlumberger_ret_1d", "heston_var_ev_h7", "ORCL_vol_20d"], "is_new": true}, {"model_id": "new_h10_STRESS_GradientBoosting_N12_t1", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 10, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWA_Australia_zscore_60d", "AXP_Amex_vol_20d", "heston_var_ev_h3", "EOG_EOGResources_ret_5d", "EWH_HongKong_ret_5d", "SBUX_vol_20d", "DAX_Germany_zscore_60d", "CMCSA_ret_1d", "PPL_PPL_ret_1d", "vix_mean_abs_ret_5d"], "is_new": true}, {"model_id": "new_h10_STRESS_GradientBoosting_N12_t2", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 10, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWQ_France_zscore_60d", "EWG_Germany_ret_20d", "MS_MorganStanley_ret_1d", "EXC_Exelon_zscore_60d", "PG_ret_20d", "EFFR_vol_20d", "DOW_Price_zscore_60d", "EWM_Malaysia_zscore_60d", "IYM_BasicMaterials_ret_20d", "EWC_Canada_zscore_60d"], "is_new": true}, {"model_id": "new_h10_STRESS_GradientBoosting_N12_t3", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 10, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "HD_ret_5d", "US3Y_Rate_ret_5d", "ITT_ITTInc_ret_5d", "EWA_Australia_ret_1d", "TXN_vol_20d", "QQQ_vol_20d", "Michigan_Sentiment_ret_20d", "BDX_Becton_Dickinson_ret_20d", "EWQ_France_ret_20d", "DHR_ret_1d"], "is_new": true}, {"model_id": "new_h10_STRESS_GradientBoosting_N12_t4", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 10, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "DIS_vol_20d", "HD_ret_1d", "DOW_Price_zscore_60d", "EWG_Germany_vol_20d", "XLY_Disc_vol_20d", "INTC_ret_5d", "ES_Evergy_ret_1d", "SJM_JM_Smucker_ret_5d", "US7Y_Rate_ret_20d", "MSTR_Bitcoin3_ret_20d"], "is_new": true}, {"model_id": "new_h10_STRESS_GradientBoosting_N12_t5", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 10, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWG_Germany_vol_20d", "DIS_vol_20d", "GE_ret_1d", "EWM_Malaysia_vol_20d", "MS_MorganStanley_ret_1d", "Industrial_Production_zscore_60d", "SCHW_Schwab_ret_5d", "Brent_Oil_FRED_ret_5d", "MS_MorganStanley_zscore_60d", "Retail_Sales_zscore_60d"], "is_new": true}, {"model_id": "new_h10_STRESS_GradientBoosting_N12_t6", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 10, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "SBUX_vol_20d", "EWM_Malaysia_vol_20d", "EWY_Korea_ret_20d", "CI_Cigna_vol_20d", "PG_ret_20d", "TGT_Target_zscore_60d", "HD_ret_20d", "spx_momentum_3d", "CCI_CrownCastle_vol_20d", "TM_Telephone_vol_20d"], "is_new": true}, {"model_id": "new_h10_STRESS_GradientBoosting_N12_t7", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 10, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "hmm_p_stress", "TED_Spread_zscore_60d", "JNJ_ret_1d", "US3M_Rate_zscore_60d", "CPB_CampbellSoup_vol_20d", "EWL_Switzerland_zscore_60d", "CPB_CampbellSoup_ret_20d", "EWY_Korea_ret_20d", "AXP_Amex_ret_20d", "EWJ_Japan_vol_20d"], "is_new": true}, {"model_id": "new_h10_STRESS_GradientBoosting_N15_t0", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 10, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "WTI_Oil_FRED_zscore_60d", "VOD_Vodafone_zscore_60d", "PAYX_Paychex_zscore_60d", "NFCI_ret_5d", "PAYX_Paychex_ret_20d", "DAX_Germany_vol_20d", "AVB_AvalonBay_zscore_60d", "MSTR_Bitcoin3_ret_5d", "EQIX_Equinix_ret_5d", "AORD_AUS_zscore_60d", "EWS_Singapore_ret_5d", "NOC_Northrop_ret_20d", "HD_ret_5d"], "is_new": true}, {"model_id": "new_h10_STRESS_GradientBoosting_N15_t1", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 10, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "Retail_Sales_zscore_60d", "PFE_ret_1d", "EWQ_France_ret_20d", "LOW_Lowes_ret_20d", "NEE_NextEra_ret_20d", "GE_ret_1d", "EXC_Exelon_zscore_60d", "US30Y_Rate_ret_20d", "FedFunds_zscore_60d", "heston_var_ev_h5", "US1Y_Rate_ret_5d", "PAYX_Paychex_vol_20d", "gjr_condvar_h1"], "is_new": true}, {"model_id": "new_h10_STRESS_GradientBoosting_N15_t2", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 10, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWQ_France_ret_20d", "US3Y_Rate_ret_5d", "gjr_condvar_h1", "PAYX_Paychex_ret_20d", "XLY_Disc_vol_20d", "AXP_Amex_ret_20d", "ORCL_vol_20d", "GD_GeneralDynamics_zscore_60d", "MO_AltriaMG_ret_1d", "spx_abs_ret_max_5d", "EXC_Exelon_zscore_60d", "Michigan_Sentiment_ret_20d", "MS_MorganStanley_zscore_60d"], "is_new": true}, {"model_id": "new_h10_STRESS_GradientBoosting_N15_t3", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 10, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "Retail_Sales_zscore_60d", "HD_ret_1d", "INTC_ret_1d", "Core_PCE_zscore_60d", "MRK_Merck_zscore_60d", "NEE_NextEra_ret_20d", "NOC_Northrop_ret_20d", "BDX_Becton_Dickinson_ret_20d", "VRP_ma5", "SCHW_Schwab_ret_5d", "SJM_JM_Smucker_ret_1d", "gjr_condvar_h1", "EWA_Australia_zscore_60d"], "is_new": true}, {"model_id": "new_h10_STRESS_GradientBoosting_N15_t4", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 10, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "BDX_Becton_Dickinson_ret_20d", "ITT_ITTInc_ret_5d", "PCAR_PaccarInc_ret_5d", "US1Y_Rate_ret_20d", "MSTR_Bitcoin3_ret_1d", "spx_momentum_3d", "FedFunds_zscore_60d", "TM_Telephone_ret_1d", "DAX_Germany_vol_20d", "AMZN_ret_5d", "SBUX_ret_5d", "EWH_HongKong_ret_5d", "SJM_JM_Smucker_ret_1d"], "is_new": true}, {"model_id": "new_h10_STRESS_GradientBoosting_N15_t5", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 10, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "VOD_Vodafone_zscore_60d", "BTI_BritishAmerican_ret_5d", "AMGN_Amgen_ret_1d", "ASX_Australia_vol_20d", "JNJ_ret_1d", "PCAR_PaccarInc_ret_5d", "XLF_Fin_vol_20d", "ORCL_vol_20d", "HUM_Humana_ret_5d", "TXN_vol_20d", "EWM_Malaysia_vol_20d", "EXC_Exelon_zscore_60d", "IBEX_Spain_ret_20d"], "is_new": true}, {"model_id": "new_h10_STRESS_GradientBoosting_N15_t6", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 10, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWQ_France_ret_20d", "PPL_PPL_ret_1d", "ORCL_vol_20d", "NOC_Northrop_ret_20d", "NWL_Newell_ret_20d", "US3M_Rate_vol_20d", "US7Y_Rate_ret_20d", "CLX_Clorox_vol_20d", "Nikkei_Japan_vol_20d", "PAYX_Paychex_vol_20d", "DE_Deere_vol_20d", "DHR_ret_1d", "HD_ret_1d"], "is_new": true}, {"model_id": "new_h10_STRESS_GradientBoosting_N15_t7", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 10, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWQ_France_ret_20d", "WTI_Oil_FRED_zscore_60d", "QQQ_vol_20d", "US1Y_Rate_ret_5d", "EWS_Singapore_ret_5d", "CCI_CrownCastle_vol_20d", "hmm_p_stress", "Brent_Oil_FRED_ret_20d", "Core_CPI_zscore_60d", "ITT_ITTInc_ret_5d", "AMGN_Amgen_ret_1d", "DE_Deere_vol_20d", "T10Y2Y_Spread_ret_5d"], "is_new": true}, {"model_id": "new_h10_STRESS_GradientBoosting_N20_t0", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 10, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "PAYX_Paychex_zscore_60d", "LUV_SouthwestAir_ret_5d", "EWQ_France_zscore_60d", "EWG_Germany_vol_20d", "EWY_Korea_zscore_60d", "ASX_Australia_ret_5d", "HUM_Humana_ret_5d", "GILD_Gilead_ret_20d", "BDX_Becton_Dickinson_ret_20d", "CPB_CampbellSoup_ret_20d", "DAX_Germany_vol_20d", "EOG_EOGResources_ret_5d", "US30Y_Rate_ret_20d", "IWM_SmallCap_vol_20d", "AXP_Amex_ret_20d", "SBUX_zscore_60d", "US3M_Rate_vol_20d", "gjr_condvar_h1"], "is_new": true}, {"model_id": "new_h10_STRESS_GradientBoosting_N20_t1", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 10, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "NOC_Northrop_ret_20d", "AMZN_ret_5d", "heston_var_ev_h7", "HD_ret_5d", "spx_vol_5d", "US3M_Rate_vol_20d", "EQR_Equity_ret_1d", "TXN_vol_20d", "HD_zscore_60d", "MSTR_Bitcoin3_ret_1d", "BTI_BritishAmerican_ret_5d", "SBUX_vol_20d", "TM_Telephone_ret_1d", "US1Y_Rate_ret_5d", "AORD_AUS_zscore_60d", "IBEX_Spain_ret_20d", "CPB_CampbellSoup_zscore_60d", "EWQ_France_ret_20d"], "is_new": true}, {"model_id": "new_h10_STRESS_GradientBoosting_N20_t2", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 10, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EXC_Exelon_zscore_60d", "EWA_Australia_ret_1d", "LOW_Lowes_ret_5d", "IYM_BasicMaterials_ret_20d", "IYR_US_REIT2_zscore_60d", "BDX_Becton_Dickinson_ret_20d", "VOD_Vodafone_zscore_60d", "spx_vol_5d", "GD_GeneralDynamics_zscore_60d", "HangSeng_HK_ret_5d", "Industrial_Production_zscore_60d", "US3M_Rate_zscore_60d", "DIS_vol_20d", "spx_abs_ret_max_5d", "ITT_ITTInc_ret_5d", "US6M_Rate_ret_20d", "TGT_Target_zscore_60d", "MS_MorganStanley_ret_1d"], "is_new": true}, {"model_id": "new_h10_STRESS_GradientBoosting_N20_t3", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 10, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "WTI_Oil_FRED_zscore_60d", "US1Y_Rate_ret_5d", "EWG_Germany_ret_20d", "XLF_Fin_vol_20d", "DE_Deere_ret_5d", "US7Y_Rate_ret_20d", "EWL_Switzerland_vol_20d", "PPL_PPL_ret_1d", "FedFunds_zscore_60d", "M_Macys_vol_20d", "NFCI_ret_5d", "EXC_Exelon_zscore_60d", "T_ret_1d", "PAYX_Paychex_ret_20d", "INTC_ret_1d", "ASX_Australia_vol_20d", "EWC_Canada_zscore_60d", "ES_Evergy_ret_1d"], "is_new": true}, {"model_id": "new_h10_STRESS_GradientBoosting_N20_t4", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 10, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "AVB_AvalonBay_zscore_60d", "LLY_zscore_60d", "PPL_PPL_ret_1d", "LMT_LockheedMartin_ret_1d", "XLV_Health_zscore_60d", "US7Y_Rate_ret_20d", "INTC_ret_1d", "VVIX_ret_20d", "MS_MorganStanley_ret_5d", "CPB_CampbellSoup_zscore_60d", "EWA_Australia_ret_1d", "DAX_Germany_vol_20d", "EOG_EOGResources_vol_20d", "TED_Spread_zscore_60d", "ASX_Australia_ret_5d", "CPB_CampbellSoup_ret_20d", "CTAS_Cintas_vol_20d", "EWH_HongKong_ret_5d"], "is_new": true}, {"model_id": "new_h10_STRESS_GradientBoosting_N20_t5", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 10, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "AVB_AvalonBay_zscore_60d", "EWC_Canada_zscore_60d", "TM_Telephone_vol_20d", "HUM_Humana_ret_5d", "NOC_Northrop_ret_20d", "QQQ_vol_20d", "MO_AltriaMG_ret_1d", "spx_abs_ret_max_5d", "PAYX_Paychex_zscore_60d", "GILD_Gilead_ret_20d", "heston_ev_h3", "DAX_Germany_vol_20d", "US6M_Rate_ret_20d", "EQIX_Equinix_ret_5d", "FedFunds_zscore_60d", "MSTR_Bitcoin3_ret_20d", "LMT_LockheedMartin_vol_20d", "XLK_Tech_zscore_60d"], "is_new": true}, {"model_id": "new_h10_STRESS_GradientBoosting_N20_t6", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 10, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "SLB_Schlumberger_ret_1d", "Brent_Oil_FRED_ret_5d", "DIS_vol_20d", "DHR_vol_20d", "EOG_EOGResources_ret_5d", "DHR_ret_1d", "US3Y_Rate_ret_5d", "BLK_BlackRock_zscore_60d", "XLB_Materials_zscore_60d", "JNJ_ret_1d", "SCHW_Schwab_ret_5d", "MO_AltriaMG_ret_1d", "EMR_Emerson_ret_20d", "PAYX_Paychex_zscore_60d", "Brent_Oil_FRED_ret_20d", "SBUX_zscore_60d", "IYM_BasicMaterials_ret_20d", "NOC_Northrop_ret_20d"], "is_new": true}, {"model_id": "new_h10_STRESS_GradientBoosting_N20_t7", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 10, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "PAYX_Paychex_zscore_60d", "PAYX_Paychex_vol_20d", "EWY_Korea_ret_20d", "EOG_EOGResources_vol_20d", "SJM_JM_Smucker_ret_5d", "NEE_NextEra_ret_20d", "HangSeng_HK_vol_20d", "ORCL_zscore_60d", "T10Y2Y_Spread_ret_5d", "gjr_condvar_h1", "MSTR_Bitcoin3_ret_1d", "US3M_Rate_vol_20d", "heston_var_ev_h7", "PAYX_Paychex_ret_20d", "Brent_Oil_FRED_ret_20d", "EQIX_Equinix_ret_5d", "SLB_Schlumberger_ret_1d", "EXC_Exelon_zscore_60d"], "is_new": true}, {"model_id": "new_h10_STRESS_GradientBoosting_N25_t0", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 10, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "FedFunds_zscore_60d", "EFFR_ret_1d", "EWQ_France_ret_20d", "DE_Deere_ret_5d", "PPL_PPL_ret_1d", "EFFR_vol_20d", "XLB_Materials_zscore_60d", "spx_abs_ret_max_5d", "AMD_ret_1d", "ES_Evergy_ret_1d", "heston_var_ev_h5", "TGT_Target_zscore_60d", "M_Macys_vol_20d", "GE_ret_1d", "US6M_Rate_ret_20d", "CTAS_Cintas_vol_20d", "EWM_Malaysia_vol_20d", "XLF_Fin_vol_20d", "Core_CPI_zscore_60d", "MS_MorganStanley_zscore_60d", "Brent_Oil_FRED_ret_20d", "HD_ret_5d", "LOW_Lowes_ret_20d"], "is_new": true}, {"model_id": "new_h10_STRESS_GradientBoosting_N25_t1", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 10, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "CTAS_Cintas_vol_20d", "INTC_ret_5d", "IYM_BasicMaterials_ret_20d", "SJM_JM_Smucker_ret_5d", "XLV_Health_zscore_60d", "US6M_Rate_ret_20d", "ES_Evergy_ret_1d", "LMT_LockheedMartin_vol_20d", "GILD_Gilead_ret_20d", "BA_ret_1d", "heston_var_ev_h7", "Nikkei_Japan_zscore_60d", "HD_ret_20d", "DHR_vol_20d", "PFE_ret_1d", "CMCSA_ret_1d", "SLB_Schlumberger_ret_1d", "NWL_Newell_ret_20d", "FedFunds_zscore_60d", "vix_mean_abs_ret_5d", "XOM_ret_1d", "LUV_SouthwestAir_ret_5d", "EWY_Korea_ret_20d"], "is_new": true}, {"model_id": "new_h10_STRESS_GradientBoosting_N25_t2", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 10, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "TED_Spread_zscore_60d", "MSTR_Bitcoin3_ret_1d", "AXP_Amex_vol_20d", "EFFR_vol_20d", "ASX_Australia_ret_5d", "EWY_Korea_zscore_60d", "3M_ret_5d", "DAX_Germany_zscore_60d", "LMT_LockheedMartin_ret_1d", "BTI_BritishAmerican_ret_5d", "EWQ_France_ret_20d", "GE_ret_1d", "LLY_zscore_60d", "PLD_Prologis_ret_5d", "US6M_Rate_ret_20d", "vix_acceleration_1d", "HD_ret_1d", "TXN_vol_20d", "SO_SouthernCo_ret_5d", "SLB_Schlumberger_ret_5d", "XLY_Disc_vol_20d", "CPB_CampbellSoup_zscore_60d", "CPB_CampbellSoup_vol_20d"], "is_new": true}, {"model_id": "new_h10_STRESS_GradientBoosting_N25_t3", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 10, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "AORD_AUS_zscore_60d", "CMCSA_ret_1d", "GD_GeneralDynamics_zscore_60d", "INTC_ret_5d", "AMD_ret_5d", "Nikkei_Japan_zscore_60d", "CI_Cigna_vol_20d", "M_Macys_vol_20d", "Brent_Oil_FRED_ret_5d", "IYM_BasicMaterials_ret_20d", "AXP_Amex_ret_20d", "SJM_JM_Smucker_ret_1d", "ES_Evergy_ret_1d", "US3M_Rate_zscore_60d", "CPB_CampbellSoup_vol_20d", "XLF_Fin_vol_20d", "NVDA_vol_20d", "BLK_BlackRock_zscore_60d", "NOC_Northrop_ret_20d", "CPB_CampbellSoup_zscore_60d", "GE_ret_1d", "HD_zscore_60d", "VOD_Vodafone_zscore_60d"], "is_new": true}, {"model_id": "new_h10_STRESS_GradientBoosting_N25_t4", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 10, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EFFR_ret_1d", "PPL_PPL_ret_1d", "PFE_ret_1d", "LUV_SouthwestAir_ret_5d", "EWY_Korea_zscore_60d", "GD_GeneralDynamics_zscore_60d", "CPB_CampbellSoup_vol_20d", "ASX_Australia_ret_5d", "US6M_Rate_ret_20d", "US5Y_Rate_ret_5d", "EWQ_France_zscore_60d", "JNJ_ret_1d", "TED_Spread_vol_20d", "LOW_Lowes_ret_5d", "vix_mean_abs_ret_5d", "DE_Deere_ret_5d", "PAYX_Paychex_zscore_60d", "HD_ret_1d", "Core_PCE_zscore_60d", "Michigan_Sentiment_ret_20d", "spx_abs_ret_max_5d", "SBUX_ret_5d", "BTI_BritishAmerican_ret_5d"], "is_new": true}, {"model_id": "new_h10_STRESS_GradientBoosting_N25_t5", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 10, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "SO_SouthernCo_ret_5d", "BTI_BritishAmerican_ret_5d", "gjr_condvar_h1", "TED_Spread_zscore_60d", "IYR_US_REIT2_zscore_60d", "SJM_JM_Smucker_ret_1d", "CMCSA_ret_1d", "MSTR_Bitcoin3_ret_5d", "EWM_Malaysia_vol_20d", "Brent_Oil_FRED_ret_5d", "AVB_AvalonBay_zscore_60d", "EWL_Switzerland_zscore_60d", "AMGN_Amgen_ret_1d", "CPB_CampbellSoup_ret_20d", "spx_vol_5d", "EWJ_Japan_vol_20d", "EWA_Australia_zscore_60d", "US3M_Rate_vol_20d", "VRP_ma5", "SBUX_vol_20d", "3M_ret_5d", "XOM_ret_20d", "IYM_BasicMaterials_ret_20d"], "is_new": true}, {"model_id": "new_h10_STRESS_GradientBoosting_N25_t6", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 10, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "XLV_Health_zscore_60d", "Industrial_Production_zscore_60d", "EWM_Malaysia_zscore_60d", "Michigan_Sentiment_ret_20d", "FedFunds_zscore_60d", "DOW_Price_zscore_60d", "BTI_BritishAmerican_ret_5d", "EWH_HongKong_ret_5d", "SBUX_vol_20d", "vix_mean_abs_ret_5d", "BTI_BritishAmerican_ret_20d", "CTAS_Cintas_vol_20d", "INTC_ret_1d", "MS_MorganStanley_ret_1d", "US6M_Rate_ret_20d", "XLB_Materials_zscore_60d", "PAYX_Paychex_zscore_60d", "CI_Cigna_vol_20d", "LOW_Lowes_ret_5d", "ASX_Australia_ret_5d", "US30Y_Rate_ret_20d", "EXC_Exelon_zscore_60d", "VVIX_ret_20d"], "is_new": true}, {"model_id": "new_h10_STRESS_GradientBoosting_N25_t7", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 10, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "XLF_Fin_vol_20d", "CPB_CampbellSoup_zscore_60d", "Retail_Sales_zscore_60d", "TED_Spread_zscore_60d", "EWM_Malaysia_ret_1d", "ORCL_zscore_60d", "HangSeng_HK_vol_20d", "MS_MorganStanley_zscore_60d", "IBEX_Spain_ret_20d", "EWS_Singapore_ret_5d", "SJM_JM_Smucker_ret_5d", "3M_ret_5d", "EWL_Switzerland_vol_20d", "NVDA_vol_20d", "PG_ret_20d", "DHR_vol_20d", "EWA_Australia_zscore_60d", "BLK_BlackRock_zscore_60d", "AVB_AvalonBay_zscore_60d", "PLD_Prologis_ret_5d", "vix_acceleration_1d", "Industrial_Production_zscore_60d", "EWY_Korea_zscore_60d"], "is_new": true}, {"model_id": "new_h10_STRESS_GradientBoosting_N30_t0", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 10, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "CPB_CampbellSoup_ret_20d", "JNJ_ret_1d", "HD_ret_5d", "SO_SouthernCo_ret_5d", "EFFR_vol_20d", "PAYX_Paychex_zscore_60d", "spx_abs_ret_max_5d", "EQR_Equity_ret_1d", "AMD_ret_1d", "AMGN_Amgen_ret_1d", "Nikkei_Japan_zscore_60d", "XLY_Disc_vol_20d", "PLD_Prologis_ret_5d", "SJM_JM_Smucker_ret_5d", "SBUX_vol_20d", "XOM_ret_20d", "HD_ret_20d", "TGT_Target_zscore_60d", "US3M_Rate_vol_20d", "BA_ret_1d", "BDX_Becton_Dickinson_ret_20d", "XLB_Materials_zscore_60d", "M_Macys_vol_20d", "EWY_Korea_zscore_60d", "Nikkei_Japan_vol_20d", "Retail_Sales_zscore_60d", "LOW_Lowes_ret_20d", "heston_var_ev_h5"], "is_new": true}, {"model_id": "new_h10_STRESS_GradientBoosting_N30_t1", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 10, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "HD_ret_20d", "EQR_Equity_ret_1d", "AORD_AUS_zscore_60d", "CPB_CampbellSoup_vol_20d", "IBEX_Spain_ret_20d", "EOG_EOGResources_vol_20d", "US6M_Rate_ret_20d", "NFCI_ret_5d", "US3M_Rate_zscore_60d", "US7Y_Rate_ret_20d", "TED_Spread_zscore_60d", "TXN_vol_20d", "JNJ_ret_1d", "PG_ret_20d", "VOD_Vodafone_zscore_60d", "EWL_Switzerland_vol_20d", "TM_Telephone_vol_20d", "EWM_Malaysia_ret_1d", "GE_ret_1d", "DAX_Germany_vol_20d", "EWM_Malaysia_vol_20d", "US1Y_Rate_ret_5d", "Core_PCE_zscore_60d", "EWL_Switzerland_zscore_60d", "CTAS_Cintas_vol_20d", "HD_ret_5d", "AMZN_ret_5d", "LUV_SouthwestAir_ret_5d"], "is_new": true}, {"model_id": "new_h10_STRESS_GradientBoosting_N30_t2", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 10, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EFFR_vol_20d", "IWM_SmallCap_vol_20d", "XLY_Disc_vol_20d", "heston_var_ev_h3", "FedFunds_zscore_60d", "AXP_Amex_vol_20d", "DAX_Germany_vol_20d", "TXN_vol_20d", "XLF_Fin_vol_20d", "SLB_Schlumberger_ret_1d", "AMT_AmericanTower_ret_1d", "MRK_Merck_zscore_60d", "EWJ_Japan_vol_20d", "SO_SouthernCo_ret_5d", "heston_ev_h3", "DHR_vol_20d", "VOD_Vodafone_zscore_60d", "SBUX_vol_20d", "DE_Deere_vol_20d", "EWH_HongKong_ret_5d", "NFCI_ret_5d", "CCI_CrownCastle_vol_20d", "Core_PCE_zscore_60d", "T_ret_1d", "IYM_BasicMaterials_ret_20d", "PCAR_PaccarInc_ret_5d", "SCHW_Schwab_ret_5d", "XLB_Materials_zscore_60d"], "is_new": true}, {"model_id": "new_h10_STRESS_GradientBoosting_N30_t3", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 10, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "spx_vol_5d", "HUM_Humana_ret_5d", "CLX_Clorox_vol_20d", "SBUX_ret_5d", "XLV_Health_zscore_60d", "US1Y_Rate_ret_20d", "FedFunds_zscore_60d", "EWL_Switzerland_vol_20d", "heston_var_ev_h5", "VRP_ma5", "US30Y_Rate_ret_20d", "EQIX_Equinix_ret_5d", "CPB_CampbellSoup_ret_5d", "ASX_Australia_vol_20d", "HangSeng_HK_ret_5d", "CTAS_Cintas_vol_20d", "US5Y_Rate_ret_5d", "ENB_EnbridgeInc_ret_1d", "EOG_EOGResources_ret_5d", "HD_ret_1d", "LOW_Lowes_ret_5d", "AMD_ret_5d", "MS_MorganStanley_ret_5d", "EXC_Exelon_ret_1d", "GD_GeneralDynamics_zscore_60d", "EQR_Equity_ret_1d", "EWQ_France_zscore_60d", "IBEX_Spain_ret_20d"], "is_new": true}, {"model_id": "new_h10_STRESS_GradientBoosting_N30_t4", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 10, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EOG_EOGResources_ret_5d", "SJM_JM_Smucker_ret_5d", "CPB_CampbellSoup_ret_5d", "Brent_Oil_FRED_ret_20d", "MS_MorganStanley_ret_5d", "SBUX_vol_20d", "IYR_US_REIT2_zscore_60d", "gjr_condvar_h1", "ASX_Australia_vol_20d", "DAX_Germany_zscore_60d", "DIS_vol_20d", "AMD_ret_5d", "VVIX_ret_20d", "XLY_Disc_vol_20d", "EQIX_Equinix_ret_5d", "EXC_Exelon_ret_1d", "spx_abs_ret_max_5d", "BA_ret_1d", "PG_ret_20d", "NOC_Northrop_ret_20d", "EQR_Equity_ret_1d", "Nikkei_Japan_vol_20d", "EWG_Germany_vol_20d", "EFFR_ret_1d", "PAYX_Paychex_zscore_60d", "EWY_Korea_zscore_60d", "US3M_Rate_zscore_60d", "DHR_ret_1d"], "is_new": true}, {"model_id": "new_h10_STRESS_GradientBoosting_N30_t5", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 10, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EOG_EOGResources_ret_5d", "TM_Telephone_ret_1d", "ITT_ITTInc_ret_5d", "EWQ_France_ret_20d", "US7Y_Rate_ret_20d", "EWS_Singapore_ret_5d", "EWG_Germany_vol_20d", "Core_CPI_zscore_60d", "CPB_CampbellSoup_ret_5d", "hmm_p_stress", "EOG_EOGResources_vol_20d", "AXP_Amex_vol_20d", "AVB_AvalonBay_zscore_60d", "US30Y_Rate_ret_20d", "DE_Deere_ret_5d", "XOM_ret_1d", "MO_AltriaMG_ret_1d", "T_ret_1d", "CPB_CampbellSoup_vol_20d", "AORD_AUS_zscore_60d", "PAYX_Paychex_ret_20d", "MSTR_Bitcoin3_ret_1d", "EWM_Malaysia_zscore_60d", "TM_Telephone_vol_20d", "CLX_Clorox_vol_20d", "EWY_Korea_ret_20d", "MSTR_Bitcoin3_ret_20d", "BDX_Becton_Dickinson_ret_20d"], "is_new": true}, {"model_id": "new_h10_STRESS_GradientBoosting_N30_t6", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 10, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "Core_CPI_zscore_60d", "NWL_Newell_ret_20d", "CPB_CampbellSoup_ret_5d", "TED_Spread_zscore_60d", "ITT_ITTInc_ret_5d", "PLD_Prologis_ret_5d", "MS_MorganStanley_ret_1d", "EFFR_ret_1d", "HangSeng_HK_ret_5d", "SCHW_Schwab_ret_5d", "DOW_Price_zscore_60d", "QQQ_vol_20d", "IYM_BasicMaterials_ret_20d", "T_ret_1d", "CCI_CrownCastle_vol_20d", "XLF_Fin_vol_20d", "ASX_Australia_vol_20d", "SPY_zscore_60d", "CLX_Clorox_vol_20d", "XOM_ret_1d", "HUM_Humana_ret_5d", "XLB_Materials_zscore_60d", "JNJ_ret_1d", "AXP_Amex_vol_20d", "AORD_AUS_zscore_60d", "EWH_HongKong_ret_5d", "Brent_Oil_FRED_ret_5d", "US6M_Rate_ret_20d"], "is_new": true}, {"model_id": "new_h10_STRESS_GradientBoosting_N30_t7", "algo": "GradientBoosting", "regime": "STRESS", "horizon": 10, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWS_Singapore_ret_5d", "CTAS_Cintas_vol_20d", "MS_MorganStanley_ret_1d", "HD_ret_5d", "AMGN_Amgen_ret_1d", "US7Y_Rate_ret_20d", "SJM_JM_Smucker_ret_1d", "IWM_SmallCap_vol_20d", "MO_AltriaMG_ret_1d", "MSTR_Bitcoin3_ret_1d", "EWJ_Japan_vol_20d", "HangSeng_HK_ret_1d", "AMT_AmericanTower_ret_1d", "spx_abs_ret_max_5d", "XLY_Disc_vol_20d", "EWC_Canada_zscore_60d", "HD_ret_20d", "XLV_Health_zscore_60d", "VVIX_ret_20d", "GD_GeneralDynamics_zscore_60d", "SLB_Schlumberger_ret_5d", "SLB_Schlumberger_ret_1d", "HangSeng_HK_ret_5d", "HangSeng_HK_vol_20d", "TGT_Target_zscore_60d", "FedFunds_zscore_60d", "ORCL_zscore_60d", "EWM_Malaysia_ret_1d"], "is_new": true}, {"model_id": "new_h10_STRESS_RandomForest_N5_t0", "algo": "RandomForest", "regime": "STRESS", "horizon": 10, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWH_HongKong_ret_5d", "EWC_Canada_zscore_60d", "US3M_Rate_vol_20d"], "is_new": true}, {"model_id": "new_h10_STRESS_RandomForest_N5_t1", "algo": "RandomForest", "regime": "STRESS", "horizon": 10, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "PAYX_Paychex_vol_20d", "Brent_Oil_FRED_ret_5d", "vix_acceleration_1d"], "is_new": true}, {"model_id": "new_h10_STRESS_RandomForest_N5_t2", "algo": "RandomForest", "regime": "STRESS", "horizon": 10, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "MS_MorganStanley_ret_5d", "EWL_Switzerland_zscore_60d", "heston_var_ev_h7"], "is_new": true}, {"model_id": "new_h10_STRESS_RandomForest_N5_t3", "algo": "RandomForest", "regime": "STRESS", "horizon": 10, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "CLX_Clorox_vol_20d", "CPB_CampbellSoup_vol_20d", "INTC_ret_1d"], "is_new": true}, {"model_id": "new_h10_STRESS_RandomForest_N5_t4", "algo": "RandomForest", "regime": "STRESS", "horizon": 10, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "DHR_vol_20d", "Retail_Sales_zscore_60d", "HangSeng_HK_ret_5d"], "is_new": true}, {"model_id": "new_h10_STRESS_RandomForest_N5_t5", "algo": "RandomForest", "regime": "STRESS", "horizon": 10, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "XLB_Materials_zscore_60d", "US3Y_Rate_ret_5d", "US1Y_Rate_ret_20d"], "is_new": true}, {"model_id": "new_h10_STRESS_RandomForest_N5_t6", "algo": "RandomForest", "regime": "STRESS", "horizon": 10, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "DOW_Price_zscore_60d", "SO_SouthernCo_ret_5d", "PCAR_PaccarInc_ret_5d"], "is_new": true}, {"model_id": "new_h10_STRESS_RandomForest_N5_t7", "algo": "RandomForest", "regime": "STRESS", "horizon": 10, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "HD_ret_1d", "DOW_Price_zscore_60d", "SO_SouthernCo_ret_5d"], "is_new": true}, {"model_id": "new_h10_STRESS_RandomForest_N8_t0", "algo": "RandomForest", "regime": "STRESS", "horizon": 10, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "XLK_Tech_zscore_60d", "EFFR_ret_1d", "CTAS_Cintas_vol_20d", "AMD_ret_1d", "EOG_EOGResources_vol_20d", "XLY_Disc_vol_20d"], "is_new": true}, {"model_id": "new_h10_STRESS_RandomForest_N8_t1", "algo": "RandomForest", "regime": "STRESS", "horizon": 10, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "DIS_vol_20d", "XLK_Tech_zscore_60d", "EWA_Australia_zscore_60d", "IWM_SmallCap_vol_20d", "IYM_BasicMaterials_ret_20d", "EWG_Germany_ret_20d"], "is_new": true}, {"model_id": "new_h10_STRESS_RandomForest_N8_t2", "algo": "RandomForest", "regime": "STRESS", "horizon": 10, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWL_Switzerland_zscore_60d", "MS_MorganStanley_zscore_60d", "XOM_ret_20d", "XLB_Materials_zscore_60d", "SO_SouthernCo_ret_5d", "LMT_LockheedMartin_ret_1d"], "is_new": true}, {"model_id": "new_h10_STRESS_RandomForest_N8_t3", "algo": "RandomForest", "regime": "STRESS", "horizon": 10, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "3M_vol_20d", "hmm_p_stress", "DE_Deere_vol_20d", "heston_var_ev_h3", "EWH_HongKong_ret_5d", "US30Y_Rate_ret_20d"], "is_new": true}, {"model_id": "new_h10_STRESS_RandomForest_N8_t4", "algo": "RandomForest", "regime": "STRESS", "horizon": 10, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "HangSeng_HK_vol_20d", "T10Y2Y_Spread_ret_5d", "BTI_BritishAmerican_ret_20d", "T_ret_1d", "CPB_CampbellSoup_zscore_60d", "HD_ret_5d"], "is_new": true}, {"model_id": "new_h10_STRESS_RandomForest_N8_t5", "algo": "RandomForest", "regime": "STRESS", "horizon": 10, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "NWL_Newell_ret_20d", "3M_ret_5d", "XOM_ret_20d", "ASX_Australia_ret_5d", "US3M_Rate_vol_20d", "AXP_Amex_ret_20d"], "is_new": true}, {"model_id": "new_h10_STRESS_RandomForest_N8_t6", "algo": "RandomForest", "regime": "STRESS", "horizon": 10, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "PLD_Prologis_ret_5d", "SJM_JM_Smucker_ret_1d", "EWY_Korea_ret_20d", "AMT_AmericanTower_ret_1d", "T_ret_1d", "PAYX_Paychex_ret_20d"], "is_new": true}, {"model_id": "new_h10_STRESS_RandomForest_N8_t7", "algo": "RandomForest", "regime": "STRESS", "horizon": 10, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EQR_Equity_ret_1d", "LOW_Lowes_ret_5d", "spx_abs_ret_max_5d", "EWQ_France_ret_20d", "TM_Telephone_ret_1d", "EWH_HongKong_ret_5d"], "is_new": true}, {"model_id": "new_h10_STRESS_RandomForest_N10_t0", "algo": "RandomForest", "regime": "STRESS", "horizon": 10, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "US5Y_Rate_ret_5d", "LUV_SouthwestAir_ret_5d", "VVIX_ret_20d", "Nikkei_Japan_zscore_60d", "DAX_Germany_zscore_60d", "EWY_Korea_ret_20d", "SCHW_Schwab_ret_5d", "ITT_ITTInc_ret_5d"], "is_new": true}, {"model_id": "new_h10_STRESS_RandomForest_N10_t1", "algo": "RandomForest", "regime": "STRESS", "horizon": 10, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "GILD_Gilead_ret_20d", "EWA_Australia_ret_1d", "HangSeng_HK_vol_20d", "EWQ_France_zscore_60d", "MSTR_Bitcoin3_ret_1d", "FedFunds_zscore_60d", "T_ret_1d", "EWM_Malaysia_ret_1d"], "is_new": true}, {"model_id": "new_h10_STRESS_RandomForest_N10_t2", "algo": "RandomForest", "regime": "STRESS", "horizon": 10, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "XLF_Fin_vol_20d", "AVB_AvalonBay_zscore_60d", "US5Y_Rate_ret_5d", "vix_acceleration_1d", "EWL_Switzerland_vol_20d", "AMGN_Amgen_ret_1d", "MO_AltriaMG_ret_1d", "heston_var_ev_h7"], "is_new": true}, {"model_id": "new_h10_STRESS_RandomForest_N10_t3", "algo": "RandomForest", "regime": "STRESS", "horizon": 10, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "PAYX_Paychex_vol_20d", "XLV_Health_zscore_60d", "CPB_CampbellSoup_ret_20d", "SJM_JM_Smucker_ret_5d", "US3Y_Rate_ret_5d", "AORD_AUS_zscore_60d", "EOG_EOGResources_vol_20d", "DE_Deere_ret_5d"], "is_new": true}, {"model_id": "new_h10_STRESS_RandomForest_N10_t4", "algo": "RandomForest", "regime": "STRESS", "horizon": 10, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "AORD_AUS_zscore_60d", "PFE_ret_1d", "SJM_JM_Smucker_ret_1d", "GILD_Gilead_ret_20d", "GD_GeneralDynamics_zscore_60d", "DHR_ret_1d", "vix_acceleration_1d", "TXN_vol_20d"], "is_new": true}, {"model_id": "new_h10_STRESS_RandomForest_N10_t5", "algo": "RandomForest", "regime": "STRESS", "horizon": 10, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "ASX_Australia_ret_5d", "spx_momentum_3d", "EWC_Canada_zscore_60d", "PCAR_PaccarInc_ret_5d", "CI_Cigna_vol_20d", "BDX_Becton_Dickinson_ret_20d", "EWL_Switzerland_zscore_60d", "SJM_JM_Smucker_ret_1d"], "is_new": true}, {"model_id": "new_h10_STRESS_RandomForest_N10_t6", "algo": "RandomForest", "regime": "STRESS", "horizon": 10, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "spx_momentum_3d", "ENB_EnbridgeInc_ret_1d", "TM_Telephone_vol_20d", "EFFR_ret_1d", "BTI_BritishAmerican_ret_5d", "PPL_PPL_ret_1d", "EWG_Germany_vol_20d", "LOW_Lowes_ret_5d"], "is_new": true}, {"model_id": "new_h10_STRESS_RandomForest_N10_t7", "algo": "RandomForest", "regime": "STRESS", "horizon": 10, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "BTI_BritishAmerican_ret_5d", "MSTR_Bitcoin3_ret_5d", "NOC_Northrop_ret_20d", "IYR_US_REIT2_zscore_60d", "PCAR_PaccarInc_ret_5d", "XLY_Disc_vol_20d", "EOG_EOGResources_vol_20d", "SBUX_vol_20d"], "is_new": true}, {"model_id": "new_h10_STRESS_RandomForest_N12_t0", "algo": "RandomForest", "regime": "STRESS", "horizon": 10, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "HangSeng_HK_ret_5d", "EWM_Malaysia_zscore_60d", "US3M_Rate_zscore_60d", "INTC_ret_1d", "XLV_Health_zscore_60d", "IYR_US_REIT2_zscore_60d", "CPB_CampbellSoup_vol_20d", "DE_Deere_ret_5d", "EWM_Malaysia_vol_20d", "NFCI_ret_5d"], "is_new": true}, {"model_id": "new_h10_STRESS_RandomForest_N12_t1", "algo": "RandomForest", "regime": "STRESS", "horizon": 10, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "GE_ret_1d", "BDX_Becton_Dickinson_ret_20d", "EWY_Korea_zscore_60d", "ASX_Australia_ret_5d", "M_Macys_vol_20d", "US3Y_Rate_ret_5d", "heston_var_ev_h7", "HangSeng_HK_vol_20d", "ITT_ITTInc_ret_5d", "QQQ_vol_20d"], "is_new": true}, {"model_id": "new_h10_STRESS_RandomForest_N12_t2", "algo": "RandomForest", "regime": "STRESS", "horizon": 10, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "heston_var_ev_h7", "TGT_Target_zscore_60d", "hmm_p_stress", "PG_ret_20d", "PLD_Prologis_ret_5d", "AMGN_Amgen_ret_1d", "heston_var_ev_h3", "ASX_Australia_vol_20d", "MSTR_Bitcoin3_ret_1d", "US7Y_Rate_ret_20d"], "is_new": true}, {"model_id": "new_h10_STRESS_RandomForest_N12_t3", "algo": "RandomForest", "regime": "STRESS", "horizon": 10, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EXC_Exelon_ret_1d", "CPB_CampbellSoup_ret_5d", "EQIX_Equinix_ret_5d", "Michigan_Sentiment_ret_20d", "TXN_vol_20d", "EWS_Singapore_ret_5d", "BTI_BritishAmerican_ret_5d", "SLB_Schlumberger_ret_5d", "XOM_ret_20d", "ES_Evergy_ret_1d"], "is_new": true}, {"model_id": "new_h10_STRESS_RandomForest_N12_t4", "algo": "RandomForest", "regime": "STRESS", "horizon": 10, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EOG_EOGResources_ret_5d", "TGT_Target_zscore_60d", "EWL_Switzerland_vol_20d", "LMT_LockheedMartin_vol_20d", "EWG_Germany_vol_20d", "CI_Cigna_vol_20d", "US1Y_Rate_ret_5d", "INTC_ret_1d", "JNJ_ret_1d", "SJM_JM_Smucker_ret_1d"], "is_new": true}, {"model_id": "new_h10_STRESS_RandomForest_N12_t5", "algo": "RandomForest", "regime": "STRESS", "horizon": 10, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "HD_zscore_60d", "PAYX_Paychex_vol_20d", "EWG_Germany_vol_20d", "SJM_JM_Smucker_ret_5d", "spx_abs_ret_max_5d", "spx_vol_5d", "DAX_Germany_zscore_60d", "EXC_Exelon_ret_1d", "US30Y_Rate_ret_20d", "GE_ret_1d"], "is_new": true}, {"model_id": "new_h10_STRESS_RandomForest_N12_t6", "algo": "RandomForest", "regime": "STRESS", "horizon": 10, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "DHR_vol_20d", "EWM_Malaysia_vol_20d", "CCI_CrownCastle_vol_20d", "hmm_p_stress", "SCHW_Schwab_ret_5d", "LLY_zscore_60d", "AMD_ret_1d", "EWY_Korea_zscore_60d", "AVB_AvalonBay_zscore_60d", "vix_mean_abs_ret_5d"], "is_new": true}, {"model_id": "new_h10_STRESS_RandomForest_N12_t7", "algo": "RandomForest", "regime": "STRESS", "horizon": 10, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "SBUX_ret_5d", "US7Y_Rate_ret_20d", "EWL_Switzerland_zscore_60d", "heston_ev_h3", "EXC_Exelon_ret_1d", "DIS_vol_20d", "3M_vol_20d", "LMT_LockheedMartin_ret_1d", "BTI_BritishAmerican_ret_20d", "heston_var_ev_h7"], "is_new": true}, {"model_id": "new_h10_STRESS_RandomForest_N15_t0", "algo": "RandomForest", "regime": "STRESS", "horizon": 10, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "MSTR_Bitcoin3_ret_5d", "EWG_Germany_ret_20d", "PG_ret_20d", "EWQ_France_zscore_60d", "BLK_BlackRock_zscore_60d", "DHR_vol_20d", "EWL_Switzerland_zscore_60d", "IYR_US_REIT2_zscore_60d", "JNJ_ret_1d", "EWA_Australia_zscore_60d", "EWA_Australia_ret_1d", "SCHW_Schwab_ret_5d", "DE_Deere_vol_20d"], "is_new": true}, {"model_id": "new_h10_STRESS_RandomForest_N15_t1", "algo": "RandomForest", "regime": "STRESS", "horizon": 10, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "AXP_Amex_ret_20d", "ES_Evergy_ret_1d", "VVIX_ret_20d", "SLB_Schlumberger_ret_1d", "EXC_Exelon_ret_1d", "CI_Cigna_vol_20d", "ITT_ITTInc_ret_5d", "3M_vol_20d", "US3Y_Rate_ret_5d", "CPB_CampbellSoup_ret_5d", "DE_Deere_vol_20d", "NEE_NextEra_ret_20d", "CPB_CampbellSoup_ret_20d"], "is_new": true}, {"model_id": "new_h10_STRESS_RandomForest_N15_t2", "algo": "RandomForest", "regime": "STRESS", "horizon": 10, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWM_Malaysia_zscore_60d", "US6M_Rate_ret_20d", "XLK_Tech_zscore_60d", "DHR_vol_20d", "3M_ret_5d", "CPB_CampbellSoup_vol_20d", "ORCL_vol_20d", "AMGN_Amgen_ret_1d", "EXC_Exelon_ret_1d", "AMT_AmericanTower_ret_1d", "HangSeng_HK_ret_1d", "EWM_Malaysia_ret_1d", "BA_ret_1d"], "is_new": true}, {"model_id": "new_h10_STRESS_RandomForest_N15_t3", "algo": "RandomForest", "regime": "STRESS", "horizon": 10, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "MSTR_Bitcoin3_ret_1d", "vix_mean_abs_ret_5d", "DE_Deere_vol_20d", "EWG_Germany_vol_20d", "DE_Deere_ret_5d", "EFFR_vol_20d", "MS_MorganStanley_ret_1d", "AORD_AUS_zscore_60d", "XLB_Materials_zscore_60d", "BTI_BritishAmerican_ret_20d", "PAYX_Paychex_ret_20d", "EWQ_France_zscore_60d", "Core_CPI_zscore_60d"], "is_new": true}, {"model_id": "new_h10_STRESS_RandomForest_N15_t4", "algo": "RandomForest", "regime": "STRESS", "horizon": 10, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "BLK_BlackRock_zscore_60d", "DE_Deere_ret_5d", "MRK_Merck_zscore_60d", "GD_GeneralDynamics_zscore_60d", "US30Y_Rate_ret_20d", "ENB_EnbridgeInc_ret_1d", "US3M_Rate_vol_20d", "hmm_p_stress", "PPL_PPL_ret_1d", "heston_var_ev_h7", "HD_zscore_60d", "ORCL_vol_20d", "gjr_condvar_h1"], "is_new": true}, {"model_id": "new_h10_STRESS_RandomForest_N15_t5", "algo": "RandomForest", "regime": "STRESS", "horizon": 10, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "PAYX_Paychex_ret_20d", "IBEX_Spain_ret_20d", "Core_PCE_zscore_60d", "BLK_BlackRock_zscore_60d", "EWM_Malaysia_zscore_60d", "AMD_ret_1d", "heston_var_ev_h7", "spx_vol_5d", "CPB_CampbellSoup_ret_20d", "EWL_Switzerland_vol_20d", "INTC_ret_5d", "US7Y_Rate_ret_20d", "XLB_Materials_zscore_60d"], "is_new": true}, {"model_id": "new_h10_STRESS_RandomForest_N15_t6", "algo": "RandomForest", "regime": "STRESS", "horizon": 10, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "heston_ev_h3", "BTI_BritishAmerican_ret_20d", "MRK_Merck_zscore_60d", "XLF_Fin_vol_20d", "LLY_zscore_60d", "SBUX_vol_20d", "MS_MorganStanley_ret_5d", "NOC_Northrop_ret_20d", "EWG_Germany_ret_20d", "IYM_BasicMaterials_ret_20d", "HD_ret_20d", "spx_momentum_3d", "vix_mean_abs_ret_5d"], "is_new": true}, {"model_id": "new_h10_STRESS_RandomForest_N15_t7", "algo": "RandomForest", "regime": "STRESS", "horizon": 10, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "HUM_Humana_ret_5d", "ASX_Australia_vol_20d", "TM_Telephone_ret_1d", "XLF_Fin_vol_20d", "NWL_Newell_ret_20d", "EFFR_vol_20d", "EWM_Malaysia_vol_20d", "AMT_AmericanTower_ret_1d", "ES_Evergy_ret_1d", "PFE_ret_1d", "CLX_Clorox_vol_20d", "GILD_Gilead_ret_20d", "EWS_Singapore_ret_5d"], "is_new": true}, {"model_id": "new_h10_STRESS_RandomForest_N20_t0", "algo": "RandomForest", "regime": "STRESS", "horizon": 10, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "DOW_Price_zscore_60d", "GILD_Gilead_ret_20d", "LMT_LockheedMartin_vol_20d", "AMZN_ret_5d", "EOG_EOGResources_vol_20d", "Industrial_Production_zscore_60d", "XLF_Fin_vol_20d", "CMCSA_ret_1d", "EQR_Equity_ret_1d", "US6M_Rate_ret_20d", "SPY_zscore_60d", "QQQ_vol_20d", "EWL_Switzerland_vol_20d", "TED_Spread_zscore_60d", "GE_ret_1d", "HUM_Humana_ret_5d", "JNJ_ret_1d", "CCI_CrownCastle_vol_20d"], "is_new": true}, {"model_id": "new_h10_STRESS_RandomForest_N20_t1", "algo": "RandomForest", "regime": "STRESS", "horizon": 10, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "Nikkei_Japan_vol_20d", "CLX_Clorox_vol_20d", "MSTR_Bitcoin3_ret_5d", "WTI_Oil_FRED_zscore_60d", "spx_vol_5d", "Retail_Sales_zscore_60d", "CPB_CampbellSoup_ret_5d", "XOM_ret_20d", "EQIX_Equinix_ret_5d", "HangSeng_HK_vol_20d", "EXC_Exelon_zscore_60d", "HD_ret_5d", "EWM_Malaysia_zscore_60d", "EWM_Malaysia_ret_1d", "US30Y_Rate_ret_20d", "DE_Deere_vol_20d", "SO_SouthernCo_ret_5d", "IYM_BasicMaterials_ret_20d"], "is_new": true}, {"model_id": "new_h10_STRESS_RandomForest_N20_t2", "algo": "RandomForest", "regime": "STRESS", "horizon": 10, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "ES_Evergy_ret_1d", "PG_ret_20d", "Brent_Oil_FRED_ret_5d", "Industrial_Production_zscore_60d", "TXN_vol_20d", "US30Y_Rate_ret_20d", "EFFR_ret_1d", "XLB_Materials_zscore_60d", "NEE_NextEra_ret_20d", "Core_CPI_zscore_60d", "CMCSA_ret_1d", "T10Y2Y_Spread_ret_5d", "LLY_zscore_60d", "AMZN_ret_5d", "CTAS_Cintas_vol_20d", "PLD_Prologis_ret_5d", "SCHW_Schwab_ret_5d", "BLK_BlackRock_zscore_60d"], "is_new": true}, {"model_id": "new_h10_STRESS_RandomForest_N20_t3", "algo": "RandomForest", "regime": "STRESS", "horizon": 10, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "GD_GeneralDynamics_zscore_60d", "EWY_Korea_zscore_60d", "LMT_LockheedMartin_ret_1d", "spx_abs_ret_max_5d", "US3M_Rate_vol_20d", "HD_ret_5d", "AMGN_Amgen_ret_1d", "M_Macys_vol_20d", "MRK_Merck_zscore_60d", "DAX_Germany_vol_20d", "DHR_ret_1d", "IWM_SmallCap_vol_20d", "US1Y_Rate_ret_20d", "CPB_CampbellSoup_vol_20d", "XOM_ret_20d", "HangSeng_HK_ret_1d", "HD_ret_20d", "Industrial_Production_zscore_60d"], "is_new": true}, {"model_id": "new_h10_STRESS_RandomForest_N20_t4", "algo": "RandomForest", "regime": "STRESS", "horizon": 10, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWY_Korea_ret_20d", "HD_ret_20d", "AMT_AmericanTower_ret_1d", "HangSeng_HK_ret_5d", "3M_ret_5d", "DAX_Germany_vol_20d", "BTI_BritishAmerican_ret_20d", "XOM_ret_20d", "EWL_Switzerland_vol_20d", "IBEX_Spain_ret_20d", "XLB_Materials_zscore_60d", "XLK_Tech_zscore_60d", "SBUX_vol_20d", "MS_MorganStanley_ret_5d", "EFFR_vol_20d", "DHR_ret_1d", "EXC_Exelon_ret_1d", "LMT_LockheedMartin_ret_1d"], "is_new": true}, {"model_id": "new_h10_STRESS_RandomForest_N20_t5", "algo": "RandomForest", "regime": "STRESS", "horizon": 10, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "spx_momentum_3d", "AMZN_ret_5d", "INTC_ret_5d", "HD_ret_20d", "ORCL_zscore_60d", "SJM_JM_Smucker_ret_5d", "US6M_Rate_ret_20d", "HangSeng_HK_vol_20d", "XOM_ret_20d", "EWA_Australia_zscore_60d", "US7Y_Rate_ret_20d", "EQIX_Equinix_ret_5d", "EWA_Australia_ret_1d", "XLF_Fin_vol_20d", "DE_Deere_ret_5d", "IYM_BasicMaterials_ret_20d", "spx_abs_ret_max_5d", "EWM_Malaysia_ret_1d"], "is_new": true}, {"model_id": "new_h10_STRESS_RandomForest_N20_t6", "algo": "RandomForest", "regime": "STRESS", "horizon": 10, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "SCHW_Schwab_ret_5d", "INTC_ret_1d", "AMD_ret_5d", "vix_acceleration_1d", "PCAR_PaccarInc_ret_5d", "SBUX_ret_5d", "heston_var_ev_h5", "ENB_EnbridgeInc_ret_1d", "SBUX_zscore_60d", "US7Y_Rate_ret_20d", "US6M_Rate_ret_20d", "US30Y_Rate_ret_20d", "SJM_JM_Smucker_ret_5d", "spx_vol_5d", "CPB_CampbellSoup_ret_20d", "US3Y_Rate_ret_5d", "CPB_CampbellSoup_zscore_60d", "Core_CPI_zscore_60d"], "is_new": true}, {"model_id": "new_h10_STRESS_RandomForest_N20_t7", "algo": "RandomForest", "regime": "STRESS", "horizon": 10, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "heston_var_ev_h5", "EWQ_France_zscore_60d", "QQQ_vol_20d", "SLB_Schlumberger_ret_5d", "XLY_Disc_vol_20d", "EQR_Equity_ret_1d", "heston_var_ev_h3", "AMD_ret_1d", "AVB_AvalonBay_zscore_60d", "CPB_CampbellSoup_ret_20d", "AMD_ret_5d", "XLB_Materials_zscore_60d", "CMCSA_ret_1d", "US6M_Rate_ret_20d", "HangSeng_HK_vol_20d", "AMGN_Amgen_ret_1d", "EWA_Australia_ret_1d", "NFCI_ret_5d"], "is_new": true}, {"model_id": "new_h10_STRESS_RandomForest_N25_t0", "algo": "RandomForest", "regime": "STRESS", "horizon": 10, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "HD_ret_1d", "XLB_Materials_zscore_60d", "TM_Telephone_ret_1d", "EMR_Emerson_ret_20d", "SCHW_Schwab_ret_5d", "EWG_Germany_ret_20d", "GE_ret_1d", "VVIX_ret_20d", "DE_Deere_ret_5d", "gjr_condvar_h1", "US3Y_Rate_ret_5d", "IBEX_Spain_ret_20d", "EXC_Exelon_zscore_60d", "NOC_Northrop_ret_20d", "FedFunds_zscore_60d", "EWG_Germany_vol_20d", "VRP_ma5", "heston_var_ev_h5", "heston_var_ev_h7", "HD_ret_20d", "QQQ_vol_20d", "TED_Spread_vol_20d", "ASX_Australia_ret_5d"], "is_new": true}, {"model_id": "new_h10_STRESS_RandomForest_N25_t1", "algo": "RandomForest", "regime": "STRESS", "horizon": 10, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "LLY_zscore_60d", "NWL_Newell_ret_20d", "vix_mean_abs_ret_5d", "HangSeng_HK_ret_1d", "MSTR_Bitcoin3_ret_1d", "EWM_Malaysia_zscore_60d", "NOC_Northrop_ret_20d", "EXC_Exelon_ret_1d", "CI_Cigna_vol_20d", "XLB_Materials_zscore_60d", "XLY_Disc_vol_20d", "NEE_NextEra_ret_20d", "XOM_ret_20d", "HUM_Humana_ret_5d", "MRK_Merck_zscore_60d", "LUV_SouthwestAir_ret_5d", "GILD_Gilead_ret_20d", "EWM_Malaysia_ret_1d", "NVDA_vol_20d", "SBUX_vol_20d", "CTAS_Cintas_vol_20d", "AXP_Amex_vol_20d", "SJM_JM_Smucker_ret_1d"], "is_new": true}, {"model_id": "new_h10_STRESS_RandomForest_N25_t2", "algo": "RandomForest", "regime": "STRESS", "horizon": 10, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "PAYX_Paychex_vol_20d", "BLK_BlackRock_zscore_60d", "CCI_CrownCastle_vol_20d", "SLB_Schlumberger_ret_5d", "CPB_CampbellSoup_ret_5d", "EQR_Equity_ret_1d", "EWM_Malaysia_ret_1d", "GD_GeneralDynamics_zscore_60d", "HangSeng_HK_ret_5d", "NOC_Northrop_ret_20d", "MSTR_Bitcoin3_ret_20d", "IYR_US_REIT2_zscore_60d", "US30Y_Rate_ret_20d", "US5Y_Rate_ret_5d", "BTI_BritishAmerican_ret_5d", "EQIX_Equinix_ret_5d", "EWG_Germany_ret_20d", "TXN_vol_20d", "QQQ_vol_20d", "CI_Cigna_vol_20d", "VRP_ma5", "PCAR_PaccarInc_ret_5d", "ES_Evergy_ret_1d"], "is_new": true}, {"model_id": "new_h10_STRESS_RandomForest_N25_t3", "algo": "RandomForest", "regime": "STRESS", "horizon": 10, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "PPL_PPL_ret_1d", "T10Y2Y_Spread_ret_5d", "HD_ret_1d", "CTAS_Cintas_vol_20d", "EFFR_ret_1d", "EWY_Korea_ret_20d", "spx_momentum_3d", "Industrial_Production_zscore_60d", "MSTR_Bitcoin3_ret_1d", "CPB_CampbellSoup_zscore_60d", "BTI_BritishAmerican_ret_5d", "XLV_Health_zscore_60d", "EWM_Malaysia_ret_1d", "EWL_Switzerland_zscore_60d", "spx_vol_5d", "TGT_Target_zscore_60d", "HangSeng_HK_vol_20d", "M_Macys_vol_20d", "XLY_Disc_vol_20d", "PAYX_Paychex_zscore_60d", "BDX_Becton_Dickinson_ret_20d", "AVB_AvalonBay_zscore_60d", "GE_ret_1d"], "is_new": true}, {"model_id": "new_h10_STRESS_RandomForest_N25_t4", "algo": "RandomForest", "regime": "STRESS", "horizon": 10, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "CPB_CampbellSoup_ret_20d", "US6M_Rate_ret_20d", "TGT_Target_zscore_60d", "AMD_ret_5d", "TED_Spread_zscore_60d", "Brent_Oil_FRED_ret_20d", "LOW_Lowes_ret_5d", "MSTR_Bitcoin3_ret_5d", "IBEX_Spain_ret_20d", "EWQ_France_ret_20d", "VRP_ma5", "EOG_EOGResources_ret_5d", "vix_mean_abs_ret_5d", "CPB_CampbellSoup_vol_20d", "VOD_Vodafone_zscore_60d", "ES_Evergy_ret_1d", "XLY_Disc_vol_20d", "EWG_Germany_vol_20d", "EWL_Switzerland_zscore_60d", "BLK_BlackRock_zscore_60d", "CTAS_Cintas_vol_20d", "BA_ret_1d", "heston_var_ev_h5"], "is_new": true}, {"model_id": "new_h10_STRESS_RandomForest_N25_t5", "algo": "RandomForest", "regime": "STRESS", "horizon": 10, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "SJM_JM_Smucker_ret_5d", "BA_ret_1d", "EWG_Germany_ret_20d", "XLF_Fin_vol_20d", "3M_ret_5d", "QQQ_vol_20d", "heston_var_ev_h3", "LMT_LockheedMartin_ret_1d", "NEE_NextEra_ret_20d", "DAX_Germany_vol_20d", "CMCSA_ret_1d", "EWL_Switzerland_zscore_60d", "SO_SouthernCo_ret_5d", "US5Y_Rate_ret_5d", "AVB_AvalonBay_zscore_60d", "HangSeng_HK_vol_20d", "BLK_BlackRock_zscore_60d", "heston_var_ev_h7", "DIS_vol_20d", "NOC_Northrop_ret_20d", "SBUX_vol_20d", "HangSeng_HK_ret_5d", "CCI_CrownCastle_vol_20d"], "is_new": true}, {"model_id": "new_h10_STRESS_RandomForest_N25_t6", "algo": "RandomForest", "regime": "STRESS", "horizon": 10, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "HD_ret_5d", "NFCI_ret_5d", "SO_SouthernCo_ret_5d", "CCI_CrownCastle_vol_20d", "Industrial_Production_zscore_60d", "WTI_Oil_FRED_zscore_60d", "IWM_SmallCap_vol_20d", "ORCL_zscore_60d", "MS_MorganStanley_zscore_60d", "XLY_Disc_vol_20d", "vix_acceleration_1d", "EMR_Emerson_ret_20d", "LMT_LockheedMartin_vol_20d", "spx_vol_5d", "MSTR_Bitcoin3_ret_1d", "NVDA_vol_20d", "XLV_Health_zscore_60d", "EWA_Australia_zscore_60d", "ASX_Australia_ret_5d", "EWY_Korea_zscore_60d", "US6M_Rate_ret_20d", "Nikkei_Japan_zscore_60d", "MSTR_Bitcoin3_ret_5d"], "is_new": true}, {"model_id": "new_h10_STRESS_RandomForest_N25_t7", "algo": "RandomForest", "regime": "STRESS", "horizon": 10, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "DOW_Price_zscore_60d", "AORD_AUS_zscore_60d", "LMT_LockheedMartin_vol_20d", "EWJ_Japan_vol_20d", "AXP_Amex_vol_20d", "MSTR_Bitcoin3_ret_5d", "HangSeng_HK_vol_20d", "US3M_Rate_vol_20d", "ASX_Australia_ret_5d", "SJM_JM_Smucker_ret_5d", "JNJ_ret_1d", "SLB_Schlumberger_ret_1d", "PFE_ret_1d", "US3Y_Rate_ret_5d", "EXC_Exelon_zscore_60d", "CMCSA_ret_1d", "HangSeng_HK_ret_1d", "SLB_Schlumberger_ret_5d", "AVB_AvalonBay_zscore_60d", "vix_acceleration_1d", "SPY_zscore_60d", "EWL_Switzerland_vol_20d", "VVIX_ret_20d"], "is_new": true}, {"model_id": "new_h10_STRESS_RandomForest_N30_t0", "algo": "RandomForest", "regime": "STRESS", "horizon": 10, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "HUM_Humana_ret_5d", "MSTR_Bitcoin3_ret_5d", "CMCSA_ret_1d", "CI_Cigna_vol_20d", "EWM_Malaysia_vol_20d", "EMR_Emerson_ret_20d", "US3Y_Rate_ret_5d", "SPY_zscore_60d", "US1Y_Rate_ret_20d", "EWJ_Japan_vol_20d", "EWG_Germany_vol_20d", "EWL_Switzerland_zscore_60d", "Nikkei_Japan_vol_20d", "DAX_Germany_vol_20d", "NEE_NextEra_ret_20d", "PAYX_Paychex_ret_20d", "VOD_Vodafone_zscore_60d", "BLK_BlackRock_zscore_60d", "AMT_AmericanTower_ret_1d", "QQQ_vol_20d", "BTI_BritishAmerican_ret_20d", "CCI_CrownCastle_vol_20d", "Retail_Sales_zscore_60d", "XLK_Tech_zscore_60d", "Industrial_Production_zscore_60d", "SLB_Schlumberger_ret_5d", "AMZN_ret_5d", "EFFR_vol_20d"], "is_new": true}, {"model_id": "new_h10_STRESS_RandomForest_N30_t1", "algo": "RandomForest", "regime": "STRESS", "horizon": 10, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EQIX_Equinix_ret_5d", "spx_vol_5d", "US3M_Rate_zscore_60d", "ASX_Australia_ret_5d", "EWJ_Japan_vol_20d", "TGT_Target_zscore_60d", "AMD_ret_5d", "NEE_NextEra_ret_20d", "US6M_Rate_ret_20d", "MO_AltriaMG_ret_1d", "CMCSA_ret_1d", "EWG_Germany_ret_20d", "gjr_condvar_h1", "XLF_Fin_vol_20d", "CTAS_Cintas_vol_20d", "CLX_Clorox_vol_20d", "SBUX_zscore_60d", "M_Macys_vol_20d", "WTI_Oil_FRED_zscore_60d", "EWM_Malaysia_ret_1d", "CPB_CampbellSoup_zscore_60d", "Core_CPI_zscore_60d", "spx_momentum_3d", "VVIX_ret_20d", "VRP_ma5", "heston_var_ev_h7", "AMGN_Amgen_ret_1d", "EOG_EOGResources_ret_5d"], "is_new": true}, {"model_id": "new_h10_STRESS_RandomForest_N30_t2", "algo": "RandomForest", "regime": "STRESS", "horizon": 10, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "PG_ret_20d", "ASX_Australia_ret_5d", "CTAS_Cintas_vol_20d", "PAYX_Paychex_vol_20d", "ORCL_vol_20d", "EWH_HongKong_ret_5d", "CCI_CrownCastle_vol_20d", "US7Y_Rate_ret_20d", "QQQ_vol_20d", "NFCI_ret_5d", "Brent_Oil_FRED_ret_20d", "EWC_Canada_zscore_60d", "T10Y2Y_Spread_ret_5d", "MS_MorganStanley_ret_5d", "US3Y_Rate_ret_5d", "BLK_BlackRock_zscore_60d", "MSTR_Bitcoin3_ret_1d", "AMT_AmericanTower_ret_1d", "Core_CPI_zscore_60d", "SJM_JM_Smucker_ret_1d", "HangSeng_HK_ret_5d", "EMR_Emerson_ret_20d", "AVB_AvalonBay_zscore_60d", "MS_MorganStanley_ret_1d", "DE_Deere_ret_5d", "TGT_Target_zscore_60d", "CMCSA_ret_1d", "EXC_Exelon_ret_1d"], "is_new": true}, {"model_id": "new_h10_STRESS_RandomForest_N30_t3", "algo": "RandomForest", "regime": "STRESS", "horizon": 10, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "US7Y_Rate_ret_20d", "MSTR_Bitcoin3_ret_1d", "DAX_Germany_zscore_60d", "LLY_zscore_60d", "GD_GeneralDynamics_zscore_60d", "HangSeng_HK_ret_5d", "US6M_Rate_ret_20d", "AORD_AUS_zscore_60d", "T_ret_1d", "SBUX_ret_5d", "US1Y_Rate_ret_5d", "AXP_Amex_ret_20d", "Brent_Oil_FRED_ret_20d", "INTC_ret_5d", "SJM_JM_Smucker_ret_5d", "SLB_Schlumberger_ret_5d", "FedFunds_zscore_60d", "heston_var_ev_h3", "AMT_AmericanTower_ret_1d", "CPB_CampbellSoup_vol_20d", "EWY_Korea_zscore_60d", "BA_ret_1d", "SLB_Schlumberger_ret_1d", "AMZN_ret_5d", "BTI_BritishAmerican_ret_5d", "DHR_ret_1d", "IYM_BasicMaterials_ret_20d", "AMD_ret_5d"], "is_new": true}, {"model_id": "new_h10_STRESS_RandomForest_N30_t4", "algo": "RandomForest", "regime": "STRESS", "horizon": 10, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "IYR_US_REIT2_zscore_60d", "BDX_Becton_Dickinson_ret_20d", "MO_AltriaMG_ret_1d", "T_ret_1d", "AXP_Amex_ret_20d", "US30Y_Rate_ret_20d", "vix_acceleration_1d", "CPB_CampbellSoup_zscore_60d", "vix_mean_abs_ret_5d", "spx_vol_5d", "INTC_ret_1d", "JNJ_ret_1d", "CLX_Clorox_vol_20d", "EWY_Korea_zscore_60d", "AXP_Amex_vol_20d", "EWC_Canada_zscore_60d", "EXC_Exelon_ret_1d", "SBUX_zscore_60d", "US3M_Rate_vol_20d", "HD_zscore_60d", "QQQ_vol_20d", "PCAR_PaccarInc_ret_5d", "MSTR_Bitcoin3_ret_1d", "HD_ret_1d", "Michigan_Sentiment_ret_20d", "EWL_Switzerland_zscore_60d", "SCHW_Schwab_ret_5d", "Retail_Sales_zscore_60d"], "is_new": true}, {"model_id": "new_h10_STRESS_RandomForest_N30_t5", "algo": "RandomForest", "regime": "STRESS", "horizon": 10, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "INTC_ret_5d", "CI_Cigna_vol_20d", "hmm_p_stress", "PLD_Prologis_ret_5d", "PAYX_Paychex_zscore_60d", "MS_MorganStanley_ret_5d", "EWQ_France_zscore_60d", "AVB_AvalonBay_zscore_60d", "HD_ret_5d", "XLF_Fin_vol_20d", "EWQ_France_ret_20d", "BTI_BritishAmerican_ret_20d", "Industrial_Production_zscore_60d", "EWA_Australia_zscore_60d", "Core_CPI_zscore_60d", "vix_acceleration_1d", "EWY_Korea_zscore_60d", "CPB_CampbellSoup_vol_20d", "DE_Deere_vol_20d", "EWS_Singapore_ret_5d", "gjr_condvar_h1", "CTAS_Cintas_vol_20d", "PPL_PPL_ret_1d", "EWJ_Japan_vol_20d", "TGT_Target_zscore_60d", "EWL_Switzerland_vol_20d", "MS_MorganStanley_zscore_60d", "HUM_Humana_ret_5d"], "is_new": true}, {"model_id": "new_h10_STRESS_RandomForest_N30_t6", "algo": "RandomForest", "regime": "STRESS", "horizon": 10, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "INTC_ret_5d", "EWM_Malaysia_ret_1d", "spx_abs_ret_max_5d", "BDX_Becton_Dickinson_ret_20d", "NWL_Newell_ret_20d", "EWA_Australia_zscore_60d", "hmm_p_stress", "QQQ_vol_20d", "SJM_JM_Smucker_ret_1d", "EWQ_France_zscore_60d", "CPB_CampbellSoup_vol_20d", "T_ret_1d", "AMD_ret_1d", "US3Y_Rate_ret_5d", "EWC_Canada_zscore_60d", "EWG_Germany_ret_20d", "EFFR_ret_1d", "TM_Telephone_ret_1d", "CCI_CrownCastle_vol_20d", "vix_acceleration_1d", "SLB_Schlumberger_ret_1d", "HangSeng_HK_ret_5d", "AORD_AUS_zscore_60d", "EWL_Switzerland_vol_20d", "US30Y_Rate_ret_20d", "EWJ_Japan_vol_20d", "spx_vol_5d", "HangSeng_HK_vol_20d"], "is_new": true}, {"model_id": "new_h10_STRESS_RandomForest_N30_t7", "algo": "RandomForest", "regime": "STRESS", "horizon": 10, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "PAYX_Paychex_vol_20d", "XOM_ret_20d", "XLB_Materials_zscore_60d", "AMD_ret_1d", "TED_Spread_vol_20d", "PFE_ret_1d", "IWM_SmallCap_vol_20d", "SBUX_zscore_60d", "PG_ret_20d", "SCHW_Schwab_ret_5d", "heston_var_ev_h3", "HangSeng_HK_ret_1d", "HangSeng_HK_vol_20d", "MSTR_Bitcoin3_ret_5d", "EOG_EOGResources_ret_5d", "3M_vol_20d", "XLY_Disc_vol_20d", "3M_ret_5d", "VOD_Vodafone_zscore_60d", "AXP_Amex_ret_20d", "CCI_CrownCastle_vol_20d", "spx_vol_5d", "BDX_Becton_Dickinson_ret_20d", "LMT_LockheedMartin_vol_20d", "TXN_vol_20d", "MO_AltriaMG_ret_1d", "XLK_Tech_zscore_60d", "CLX_Clorox_vol_20d"], "is_new": true}, {"model_id": "new_h10_STRESS_LogisticRegression_N5_t0", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 10, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "DOW_Price_zscore_60d", "DHR_vol_20d", "Retail_Sales_zscore_60d"], "is_new": true}, {"model_id": "new_h10_STRESS_LogisticRegression_N5_t1", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 10, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "T_ret_1d", "GE_ret_1d", "PAYX_Paychex_zscore_60d"], "is_new": true}, {"model_id": "new_h10_STRESS_LogisticRegression_N5_t2", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 10, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "PFE_ret_1d", "EOG_EOGResources_vol_20d", "vix_mean_abs_ret_5d"], "is_new": true}, {"model_id": "new_h10_STRESS_LogisticRegression_N5_t3", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 10, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "Brent_Oil_FRED_ret_20d", "EOG_EOGResources_vol_20d", "gjr_condvar_h1"], "is_new": true}, {"model_id": "new_h10_STRESS_LogisticRegression_N5_t4", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 10, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "TED_Spread_zscore_60d", "3M_vol_20d", "PAYX_Paychex_ret_20d"], "is_new": true}, {"model_id": "new_h10_STRESS_LogisticRegression_N5_t5", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 10, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "WTI_Oil_FRED_zscore_60d", "BA_ret_1d", "BTI_BritishAmerican_ret_20d"], "is_new": true}, {"model_id": "new_h10_STRESS_LogisticRegression_N5_t6", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 10, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "US1Y_Rate_ret_20d", "ASX_Australia_vol_20d", "SBUX_vol_20d"], "is_new": true}, {"model_id": "new_h10_STRESS_LogisticRegression_N5_t7", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 10, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "gjr_condvar_h1", "LOW_Lowes_ret_5d", "INTC_ret_1d"], "is_new": true}, {"model_id": "new_h10_STRESS_LogisticRegression_N8_t0", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 10, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "Michigan_Sentiment_ret_20d", "T_ret_1d", "HD_ret_20d", "ORCL_zscore_60d", "SLB_Schlumberger_ret_5d", "AVB_AvalonBay_zscore_60d"], "is_new": true}, {"model_id": "new_h10_STRESS_LogisticRegression_N8_t1", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 10, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EQR_Equity_ret_1d", "HD_ret_5d", "heston_ev_h3", "Industrial_Production_zscore_60d", "T10Y2Y_Spread_ret_5d", "VOD_Vodafone_zscore_60d"], "is_new": true}, {"model_id": "new_h10_STRESS_LogisticRegression_N8_t2", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 10, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "gjr_condvar_h1", "GD_GeneralDynamics_zscore_60d", "IYR_US_REIT2_zscore_60d", "EWC_Canada_zscore_60d", "PPL_PPL_ret_1d", "INTC_ret_5d"], "is_new": true}, {"model_id": "new_h10_STRESS_LogisticRegression_N8_t3", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 10, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "MSTR_Bitcoin3_ret_1d", "MSTR_Bitcoin3_ret_20d", "BTI_BritishAmerican_ret_5d", "AVB_AvalonBay_zscore_60d", "WTI_Oil_FRED_zscore_60d", "XLF_Fin_vol_20d"], "is_new": true}, {"model_id": "new_h10_STRESS_LogisticRegression_N8_t4", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 10, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "LUV_SouthwestAir_ret_5d", "EWQ_France_ret_20d", "VOD_Vodafone_zscore_60d", "T_ret_1d", "CLX_Clorox_vol_20d", "spx_momentum_3d"], "is_new": true}, {"model_id": "new_h10_STRESS_LogisticRegression_N8_t5", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 10, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "Retail_Sales_zscore_60d", "LMT_LockheedMartin_vol_20d", "XOM_ret_1d", "US5Y_Rate_ret_5d", "EFFR_ret_1d", "SBUX_zscore_60d"], "is_new": true}, {"model_id": "new_h10_STRESS_LogisticRegression_N8_t6", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 10, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "VOD_Vodafone_zscore_60d", "EWA_Australia_zscore_60d", "HD_zscore_60d", "HangSeng_HK_vol_20d", "EFFR_vol_20d", "EWH_HongKong_ret_5d"], "is_new": true}, {"model_id": "new_h10_STRESS_LogisticRegression_N8_t7", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 10, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EMR_Emerson_ret_20d", "EWL_Switzerland_zscore_60d", "DE_Deere_vol_20d", "EWQ_France_zscore_60d", "MSTR_Bitcoin3_ret_5d", "BTI_BritishAmerican_ret_20d"], "is_new": true}, {"model_id": "new_h10_STRESS_LogisticRegression_N10_t0", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 10, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWY_Korea_zscore_60d", "CCI_CrownCastle_vol_20d", "VRP_ma5", "GILD_Gilead_ret_20d", "HD_ret_20d", "ORCL_vol_20d", "Nikkei_Japan_zscore_60d", "HD_ret_5d"], "is_new": true}, {"model_id": "new_h10_STRESS_LogisticRegression_N10_t1", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 10, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "AMZN_ret_5d", "hmm_p_stress", "SBUX_vol_20d", "MO_AltriaMG_ret_1d", "ENB_EnbridgeInc_ret_1d", "DE_Deere_vol_20d", "CTAS_Cintas_vol_20d", "TGT_Target_zscore_60d"], "is_new": true}, {"model_id": "new_h10_STRESS_LogisticRegression_N10_t2", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 10, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "AMT_AmericanTower_ret_1d", "MSTR_Bitcoin3_ret_20d", "IYR_US_REIT2_zscore_60d", "MSTR_Bitcoin3_ret_5d", "PCAR_PaccarInc_ret_5d", "AMZN_ret_5d", "heston_var_ev_h5", "EMR_Emerson_ret_20d"], "is_new": true}, {"model_id": "new_h10_STRESS_LogisticRegression_N10_t3", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 10, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "CCI_CrownCastle_vol_20d", "EXC_Exelon_zscore_60d", "XLB_Materials_zscore_60d", "XLF_Fin_vol_20d", "T10Y2Y_Spread_ret_5d", "heston_var_ev_h7", "CMCSA_ret_1d", "EMR_Emerson_ret_20d"], "is_new": true}, {"model_id": "new_h10_STRESS_LogisticRegression_N10_t4", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 10, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "SLB_Schlumberger_ret_1d", "SLB_Schlumberger_ret_5d", "EWQ_France_zscore_60d", "vix_acceleration_1d", "heston_ev_h3", "XLK_Tech_zscore_60d", "BDX_Becton_Dickinson_ret_20d", "NVDA_vol_20d"], "is_new": true}, {"model_id": "new_h10_STRESS_LogisticRegression_N10_t5", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 10, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EQIX_Equinix_ret_5d", "SCHW_Schwab_ret_5d", "AVB_AvalonBay_zscore_60d", "NFCI_ret_5d", "ENB_EnbridgeInc_ret_1d", "Brent_Oil_FRED_ret_5d", "US7Y_Rate_ret_20d", "US30Y_Rate_ret_20d"], "is_new": true}, {"model_id": "new_h10_STRESS_LogisticRegression_N10_t6", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 10, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "AXP_Amex_ret_20d", "DAX_Germany_zscore_60d", "BTI_BritishAmerican_ret_20d", "GD_GeneralDynamics_zscore_60d", "INTC_ret_5d", "XOM_ret_20d", "AMD_ret_5d", "LUV_SouthwestAir_ret_5d"], "is_new": true}, {"model_id": "new_h10_STRESS_LogisticRegression_N10_t7", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 10, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "TM_Telephone_ret_1d", "US3M_Rate_vol_20d", "DE_Deere_ret_5d", "CCI_CrownCastle_vol_20d", "GE_ret_1d", "CI_Cigna_vol_20d", "LUV_SouthwestAir_ret_5d", "LMT_LockheedMartin_vol_20d"], "is_new": true}, {"model_id": "new_h10_STRESS_LogisticRegression_N12_t0", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 10, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWC_Canada_zscore_60d", "CPB_CampbellSoup_zscore_60d", "PFE_ret_1d", "GE_ret_1d", "SBUX_ret_5d", "AMD_ret_1d", "vix_mean_abs_ret_5d", "LOW_Lowes_ret_20d", "US3M_Rate_zscore_60d", "spx_momentum_3d"], "is_new": true}, {"model_id": "new_h10_STRESS_LogisticRegression_N12_t1", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 10, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "Brent_Oil_FRED_ret_5d", "ORCL_zscore_60d", "LLY_zscore_60d", "AMT_AmericanTower_ret_1d", "HD_ret_5d", "Nikkei_Japan_zscore_60d", "LUV_SouthwestAir_ret_5d", "TM_Telephone_ret_1d", "GILD_Gilead_ret_20d", "PFE_ret_1d"], "is_new": true}, {"model_id": "new_h10_STRESS_LogisticRegression_N12_t2", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 10, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "DE_Deere_ret_5d", "AMD_ret_1d", "DHR_ret_1d", "CPB_CampbellSoup_ret_5d", "EQIX_Equinix_ret_5d", "spx_momentum_3d", "US3Y_Rate_ret_5d", "BDX_Becton_Dickinson_ret_20d", "Retail_Sales_zscore_60d", "INTC_ret_5d"], "is_new": true}, {"model_id": "new_h10_STRESS_LogisticRegression_N12_t3", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 10, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "NFCI_ret_5d", "LOW_Lowes_ret_5d", "PFE_ret_1d", "SBUX_ret_5d", "Brent_Oil_FRED_ret_20d", "EFFR_ret_1d", "ENB_EnbridgeInc_ret_1d", "WTI_Oil_FRED_zscore_60d", "AXP_Amex_ret_20d", "PPL_PPL_ret_1d"], "is_new": true}, {"model_id": "new_h10_STRESS_LogisticRegression_N12_t4", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 10, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "T10Y2Y_Spread_ret_5d", "LLY_zscore_60d", "Industrial_Production_zscore_60d", "heston_var_ev_h3", "SJM_JM_Smucker_ret_5d", "DE_Deere_vol_20d", "EWY_Korea_ret_20d", "EOG_EOGResources_ret_5d", "EWM_Malaysia_vol_20d", "SLB_Schlumberger_ret_5d"], "is_new": true}, {"model_id": "new_h10_STRESS_LogisticRegression_N12_t5", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 10, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "CTAS_Cintas_vol_20d", "US5Y_Rate_ret_5d", "3M_vol_20d", "EWY_Korea_ret_20d", "EWQ_France_ret_20d", "DE_Deere_vol_20d", "QQQ_vol_20d", "CMCSA_ret_1d", "PAYX_Paychex_ret_20d", "MSTR_Bitcoin3_ret_5d"], "is_new": true}, {"model_id": "new_h10_STRESS_LogisticRegression_N12_t6", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 10, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "Nikkei_Japan_vol_20d", "EWH_HongKong_ret_5d", "HangSeng_HK_ret_1d", "DHR_ret_1d", "EWG_Germany_vol_20d", "ASX_Australia_ret_5d", "XOM_ret_20d", "WTI_Oil_FRED_zscore_60d", "PAYX_Paychex_zscore_60d", "T10Y2Y_Spread_ret_5d"], "is_new": true}, {"model_id": "new_h10_STRESS_LogisticRegression_N12_t7", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 10, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EXC_Exelon_zscore_60d", "GE_ret_1d", "SO_SouthernCo_ret_5d", "TM_Telephone_vol_20d", "AVB_AvalonBay_zscore_60d", "SBUX_ret_5d", "EWM_Malaysia_vol_20d", "MS_MorganStanley_ret_1d", "BTI_BritishAmerican_ret_20d", "EQR_Equity_ret_1d"], "is_new": true}, {"model_id": "new_h10_STRESS_LogisticRegression_N15_t0", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 10, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "Michigan_Sentiment_ret_20d", "EWA_Australia_zscore_60d", "US5Y_Rate_ret_5d", "TED_Spread_zscore_60d", "EWC_Canada_zscore_60d", "MRK_Merck_zscore_60d", "BLK_BlackRock_zscore_60d", "EXC_Exelon_zscore_60d", "XLB_Materials_zscore_60d", "EWL_Switzerland_zscore_60d", "MS_MorganStanley_ret_1d", "QQQ_vol_20d", "PAYX_Paychex_vol_20d"], "is_new": true}, {"model_id": "new_h10_STRESS_LogisticRegression_N15_t1", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 10, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "PPL_PPL_ret_1d", "DE_Deere_ret_5d", "ORCL_vol_20d", "DOW_Price_zscore_60d", "NEE_NextEra_ret_20d", "AXP_Amex_vol_20d", "MRK_Merck_zscore_60d", "NWL_Newell_ret_20d", "BTI_BritishAmerican_ret_5d", "GE_ret_1d", "Core_PCE_zscore_60d", "US1Y_Rate_ret_5d", "XLV_Health_zscore_60d"], "is_new": true}, {"model_id": "new_h10_STRESS_LogisticRegression_N15_t2", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 10, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "gjr_condvar_h1", "BTI_BritishAmerican_ret_20d", "CLX_Clorox_vol_20d", "US6M_Rate_ret_20d", "heston_ev_h3", "NEE_NextEra_ret_20d", "ENB_EnbridgeInc_ret_1d", "US3M_Rate_zscore_60d", "US7Y_Rate_ret_20d", "TED_Spread_zscore_60d", "vix_mean_abs_ret_5d", "IBEX_Spain_ret_20d", "EFFR_ret_1d"], "is_new": true}, {"model_id": "new_h10_STRESS_LogisticRegression_N15_t3", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 10, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWM_Malaysia_zscore_60d", "XLB_Materials_zscore_60d", "PAYX_Paychex_vol_20d", "heston_var_ev_h5", "CPB_CampbellSoup_ret_20d", "XLK_Tech_zscore_60d", "Core_CPI_zscore_60d", "ASX_Australia_ret_5d", "EWQ_France_zscore_60d", "EWL_Switzerland_zscore_60d", "SJM_JM_Smucker_ret_5d", "US3Y_Rate_ret_5d", "EMR_Emerson_ret_20d"], "is_new": true}, {"model_id": "new_h10_STRESS_LogisticRegression_N15_t4", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 10, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "DAX_Germany_vol_20d", "XOM_ret_1d", "PLD_Prologis_ret_5d", "CCI_CrownCastle_vol_20d", "AMZN_ret_5d", "EFFR_ret_1d", "heston_var_ev_h7", "HD_zscore_60d", "IYR_US_REIT2_zscore_60d", "Brent_Oil_FRED_ret_20d", "US6M_Rate_ret_20d", "CTAS_Cintas_vol_20d", "Retail_Sales_zscore_60d"], "is_new": true}, {"model_id": "new_h10_STRESS_LogisticRegression_N15_t5", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 10, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "US5Y_Rate_ret_5d", "PFE_ret_1d", "FedFunds_zscore_60d", "spx_momentum_3d", "NOC_Northrop_ret_20d", "EOG_EOGResources_ret_5d", "PG_ret_20d", "US3Y_Rate_ret_5d", "US1Y_Rate_ret_20d", "TXN_vol_20d", "PAYX_Paychex_vol_20d", "vix_mean_abs_ret_5d", "XLK_Tech_zscore_60d"], "is_new": true}, {"model_id": "new_h10_STRESS_LogisticRegression_N15_t6", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 10, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "US5Y_Rate_ret_5d", "XOM_ret_20d", "ASX_Australia_ret_5d", "MSTR_Bitcoin3_ret_5d", "PAYX_Paychex_zscore_60d", "MS_MorganStanley_ret_1d", "IYR_US_REIT2_zscore_60d", "EWG_Germany_vol_20d", "EWQ_France_ret_20d", "M_Macys_vol_20d", "EWM_Malaysia_ret_1d", "EWS_Singapore_ret_5d", "AMT_AmericanTower_ret_1d"], "is_new": true}, {"model_id": "new_h10_STRESS_LogisticRegression_N15_t7", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 10, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EQR_Equity_ret_1d", "HangSeng_HK_ret_5d", "NVDA_vol_20d", "PG_ret_20d", "HD_ret_1d", "spx_vol_5d", "MSTR_Bitcoin3_ret_20d", "TED_Spread_zscore_60d", "SO_SouthernCo_ret_5d", "LUV_SouthwestAir_ret_5d", "ASX_Australia_ret_5d", "CPB_CampbellSoup_ret_5d", "ORCL_vol_20d"], "is_new": true}, {"model_id": "new_h10_STRESS_LogisticRegression_N20_t0", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 10, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "SO_SouthernCo_ret_5d", "CTAS_Cintas_vol_20d", "INTC_ret_1d", "DE_Deere_vol_20d", "Core_PCE_zscore_60d", "EWY_Korea_ret_20d", "SPY_zscore_60d", "AXP_Amex_vol_20d", "EOG_EOGResources_vol_20d", "EQIX_Equinix_ret_5d", "IYR_US_REIT2_zscore_60d", "hmm_p_stress", "Brent_Oil_FRED_ret_20d", "SBUX_vol_20d", "LOW_Lowes_ret_20d", "XLV_Health_zscore_60d", "GILD_Gilead_ret_20d", "CCI_CrownCastle_vol_20d"], "is_new": true}, {"model_id": "new_h10_STRESS_LogisticRegression_N20_t1", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 10, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWY_Korea_zscore_60d", "GD_GeneralDynamics_zscore_60d", "IYR_US_REIT2_zscore_60d", "IYM_BasicMaterials_ret_20d", "AMGN_Amgen_ret_1d", "BA_ret_1d", "XLY_Disc_vol_20d", "TXN_vol_20d", "MSTR_Bitcoin3_ret_1d", "CPB_CampbellSoup_ret_5d", "VVIX_ret_20d", "BTI_BritishAmerican_ret_20d", "T_ret_1d", "US1Y_Rate_ret_20d", "ASX_Australia_vol_20d", "DHR_ret_1d", "INTC_ret_1d", "GILD_Gilead_ret_20d"], "is_new": true}, {"model_id": "new_h10_STRESS_LogisticRegression_N20_t2", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 10, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "SBUX_ret_5d", "vix_acceleration_1d", "XLY_Disc_vol_20d", "3M_ret_5d", "IWM_SmallCap_vol_20d", "ITT_ITTInc_ret_5d", "IBEX_Spain_ret_20d", "XLK_Tech_zscore_60d", "M_Macys_vol_20d", "ASX_Australia_ret_5d", "CTAS_Cintas_vol_20d", "INTC_ret_5d", "INTC_ret_1d", "US6M_Rate_ret_20d", "SLB_Schlumberger_ret_5d", "spx_momentum_3d", "EFFR_vol_20d", "SCHW_Schwab_ret_5d"], "is_new": true}, {"model_id": "new_h10_STRESS_LogisticRegression_N20_t3", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 10, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWS_Singapore_ret_5d", "EWM_Malaysia_vol_20d", "EWA_Australia_zscore_60d", "LUV_SouthwestAir_ret_5d", "NOC_Northrop_ret_20d", "INTC_ret_5d", "US5Y_Rate_ret_5d", "HD_zscore_60d", "heston_var_ev_h7", "SO_SouthernCo_ret_5d", "TM_Telephone_ret_1d", "HD_ret_1d", "SBUX_vol_20d", "LLY_zscore_60d", "IYM_BasicMaterials_ret_20d", "MO_AltriaMG_ret_1d", "EXC_Exelon_zscore_60d", "CPB_CampbellSoup_ret_20d"], "is_new": true}, {"model_id": "new_h10_STRESS_LogisticRegression_N20_t4", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 10, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "HUM_Humana_ret_5d", "WTI_Oil_FRED_zscore_60d", "HD_zscore_60d", "EWM_Malaysia_vol_20d", "INTC_ret_1d", "EFFR_ret_1d", "XLY_Disc_vol_20d", "DIS_vol_20d", "EWY_Korea_ret_20d", "BDX_Becton_Dickinson_ret_20d", "hmm_p_stress", "INTC_ret_5d", "Michigan_Sentiment_ret_20d", "BTI_BritishAmerican_ret_20d", "gjr_condvar_h1", "CPB_CampbellSoup_ret_5d", "LMT_LockheedMartin_vol_20d", "EQIX_Equinix_ret_5d"], "is_new": true}, {"model_id": "new_h10_STRESS_LogisticRegression_N20_t5", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 10, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWH_HongKong_ret_5d", "SPY_zscore_60d", "EOG_EOGResources_vol_20d", "3M_ret_5d", "DIS_vol_20d", "M_Macys_vol_20d", "EXC_Exelon_ret_1d", "AMZN_ret_5d", "PLD_Prologis_ret_5d", "NFCI_ret_5d", "LMT_LockheedMartin_vol_20d", "LLY_zscore_60d", "TXN_vol_20d", "ASX_Australia_vol_20d", "EWM_Malaysia_zscore_60d", "XOM_ret_1d", "TED_Spread_zscore_60d", "IYM_BasicMaterials_ret_20d"], "is_new": true}, {"model_id": "new_h10_STRESS_LogisticRegression_N20_t6", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 10, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "BLK_BlackRock_zscore_60d", "AMD_ret_5d", "3M_vol_20d", "ES_Evergy_ret_1d", "EWG_Germany_vol_20d", "HD_ret_5d", "CLX_Clorox_vol_20d", "DIS_vol_20d", "EWG_Germany_ret_20d", "LLY_zscore_60d", "EWJ_Japan_vol_20d", "MO_AltriaMG_ret_1d", "LMT_LockheedMartin_vol_20d", "gjr_condvar_h1", "PAYX_Paychex_ret_20d", "LOW_Lowes_ret_5d", "CPB_CampbellSoup_zscore_60d", "HD_zscore_60d"], "is_new": true}, {"model_id": "new_h10_STRESS_LogisticRegression_N20_t7", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 10, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWQ_France_zscore_60d", "TXN_vol_20d", "EWQ_France_ret_20d", "EWM_Malaysia_vol_20d", "SBUX_ret_5d", "WTI_Oil_FRED_zscore_60d", "AMD_ret_5d", "SLB_Schlumberger_ret_5d", "GD_GeneralDynamics_zscore_60d", "MSTR_Bitcoin3_ret_20d", "US7Y_Rate_ret_20d", "NOC_Northrop_ret_20d", "PAYX_Paychex_ret_20d", "INTC_ret_1d", "HD_ret_20d", "INTC_ret_5d", "SCHW_Schwab_ret_5d", "Nikkei_Japan_zscore_60d"], "is_new": true}, {"model_id": "new_h10_STRESS_LogisticRegression_N25_t0", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 10, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "T_ret_1d", "NFCI_ret_5d", "PG_ret_20d", "spx_momentum_3d", "NVDA_vol_20d", "LMT_LockheedMartin_vol_20d", "CPB_CampbellSoup_zscore_60d", "CPB_CampbellSoup_ret_5d", "XOM_ret_20d", "CI_Cigna_vol_20d", "SBUX_vol_20d", "Brent_Oil_FRED_ret_5d", "heston_var_ev_h7", "vix_mean_abs_ret_5d", "CCI_CrownCastle_vol_20d", "EOG_EOGResources_vol_20d", "WTI_Oil_FRED_zscore_60d", "SLB_Schlumberger_ret_1d", "SJM_JM_Smucker_ret_1d", "BTI_BritishAmerican_ret_5d", "MSTR_Bitcoin3_ret_20d", "HD_ret_1d", "LOW_Lowes_ret_5d"], "is_new": true}, {"model_id": "new_h10_STRESS_LogisticRegression_N25_t1", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 10, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "DE_Deere_vol_20d", "HD_ret_5d", "HD_ret_20d", "EWS_Singapore_ret_5d", "DHR_ret_1d", "EMR_Emerson_ret_20d", "ENB_EnbridgeInc_ret_1d", "AMT_AmericanTower_ret_1d", "XOM_ret_20d", "TED_Spread_zscore_60d", "SBUX_vol_20d", "PAYX_Paychex_zscore_60d", "LLY_zscore_60d", "HD_zscore_60d", "M_Macys_vol_20d", "DAX_Germany_zscore_60d", "US7Y_Rate_ret_20d", "MSTR_Bitcoin3_ret_1d", "VVIX_ret_20d", "EWL_Switzerland_vol_20d", "HUM_Humana_ret_5d", "heston_var_ev_h3", "EOG_EOGResources_vol_20d"], "is_new": true}, {"model_id": "new_h10_STRESS_LogisticRegression_N25_t2", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 10, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "DIS_vol_20d", "CTAS_Cintas_vol_20d", "EWY_Korea_zscore_60d", "LLY_zscore_60d", "Brent_Oil_FRED_ret_20d", "AXP_Amex_vol_20d", "ASX_Australia_vol_20d", "WTI_Oil_FRED_zscore_60d", "SBUX_zscore_60d", "EWH_HongKong_ret_5d", "IWM_SmallCap_vol_20d", "EQIX_Equinix_ret_5d", "AMD_ret_5d", "heston_var_ev_h7", "XLY_Disc_vol_20d", "MS_MorganStanley_ret_1d", "EWA_Australia_zscore_60d", "EWC_Canada_zscore_60d", "EWS_Singapore_ret_5d", "US1Y_Rate_ret_5d", "TM_Telephone_vol_20d", "PPL_PPL_ret_1d", "AMD_ret_1d"], "is_new": true}, {"model_id": "new_h10_STRESS_LogisticRegression_N25_t3", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 10, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "CI_Cigna_vol_20d", "EWJ_Japan_vol_20d", "Core_CPI_zscore_60d", "XLY_Disc_vol_20d", "US1Y_Rate_ret_5d", "MRK_Merck_zscore_60d", "BLK_BlackRock_zscore_60d", "HD_ret_1d", "hmm_p_stress", "DHR_ret_1d", "AMGN_Amgen_ret_1d", "LLY_zscore_60d", "NFCI_ret_5d", "CPB_CampbellSoup_ret_5d", "TXN_vol_20d", "PG_ret_20d", "HD_ret_5d", "CMCSA_ret_1d", "XLV_Health_zscore_60d", "LMT_LockheedMartin_vol_20d", "DAX_Germany_vol_20d", "VVIX_ret_20d", "HUM_Humana_ret_5d"], "is_new": true}, {"model_id": "new_h10_STRESS_LogisticRegression_N25_t4", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 10, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "SBUX_vol_20d", "EWY_Korea_ret_20d", "vix_acceleration_1d", "SJM_JM_Smucker_ret_1d", "LMT_LockheedMartin_ret_1d", "AORD_AUS_zscore_60d", "XOM_ret_1d", "SPY_zscore_60d", "Retail_Sales_zscore_60d", "BTI_BritishAmerican_ret_5d", "BDX_Becton_Dickinson_ret_20d", "IBEX_Spain_ret_20d", "DHR_ret_1d", "SO_SouthernCo_ret_5d", "TED_Spread_vol_20d", "EWG_Germany_vol_20d", "DE_Deere_ret_5d", "EOG_EOGResources_ret_5d", "EOG_EOGResources_vol_20d", "PLD_Prologis_ret_5d", "HD_ret_20d", "PFE_ret_1d", "PG_ret_20d"], "is_new": true}, {"model_id": "new_h10_STRESS_LogisticRegression_N25_t5", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 10, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWM_Malaysia_ret_1d", "IBEX_Spain_ret_20d", "SJM_JM_Smucker_ret_5d", "EOG_EOGResources_vol_20d", "T10Y2Y_Spread_ret_5d", "XLV_Health_zscore_60d", "XLY_Disc_vol_20d", "BTI_BritishAmerican_ret_20d", "CPB_CampbellSoup_ret_20d", "TGT_Target_zscore_60d", "EMR_Emerson_ret_20d", "SBUX_vol_20d", "ORCL_vol_20d", "PG_ret_20d", "spx_momentum_3d", "ENB_EnbridgeInc_ret_1d", "HD_ret_1d", "NOC_Northrop_ret_20d", "AMT_AmericanTower_ret_1d", "CTAS_Cintas_vol_20d", "VRP_ma5", "EWY_Korea_ret_20d", "WTI_Oil_FRED_zscore_60d"], "is_new": true}, {"model_id": "new_h10_STRESS_LogisticRegression_N25_t6", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 10, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "AMD_ret_1d", "US5Y_Rate_ret_5d", "HangSeng_HK_ret_5d", "MRK_Merck_zscore_60d", "DHR_ret_1d", "DOW_Price_zscore_60d", "IWM_SmallCap_vol_20d", "EWM_Malaysia_zscore_60d", "NVDA_vol_20d", "Core_CPI_zscore_60d", "GILD_Gilead_ret_20d", "SCHW_Schwab_ret_5d", "NFCI_ret_5d", "TXN_vol_20d", "HD_ret_5d", "ENB_EnbridgeInc_ret_1d", "TGT_Target_zscore_60d", "EWJ_Japan_vol_20d", "TM_Telephone_ret_1d", "US1Y_Rate_ret_20d", "T_ret_1d", "EWA_Australia_zscore_60d", "AMT_AmericanTower_ret_1d"], "is_new": true}, {"model_id": "new_h10_STRESS_LogisticRegression_N25_t7", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 10, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWL_Switzerland_zscore_60d", "HD_ret_20d", "PG_ret_20d", "3M_vol_20d", "INTC_ret_1d", "DE_Deere_vol_20d", "HangSeng_HK_ret_5d", "vix_mean_abs_ret_5d", "EWY_Korea_zscore_60d", "IYM_BasicMaterials_ret_20d", "ENB_EnbridgeInc_ret_1d", "US7Y_Rate_ret_20d", "gjr_condvar_h1", "PPL_PPL_ret_1d", "US30Y_Rate_ret_20d", "MS_MorganStanley_ret_1d", "heston_var_ev_h5", "DOW_Price_zscore_60d", "BTI_BritishAmerican_ret_20d", "WTI_Oil_FRED_zscore_60d", "SO_SouthernCo_ret_5d", "IYR_US_REIT2_zscore_60d", "BTI_BritishAmerican_ret_5d"], "is_new": true}, {"model_id": "new_h10_STRESS_LogisticRegression_N30_t0", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 10, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "US3M_Rate_zscore_60d", "T_ret_1d", "EWA_Australia_zscore_60d", "EXC_Exelon_zscore_60d", "EWG_Germany_ret_20d", "Retail_Sales_zscore_60d", "SPY_zscore_60d", "GD_GeneralDynamics_zscore_60d", "INTC_ret_5d", "EWY_Korea_ret_20d", "SBUX_ret_5d", "HangSeng_HK_ret_1d", "TED_Spread_zscore_60d", "EWL_Switzerland_vol_20d", "HUM_Humana_ret_5d", "XLV_Health_zscore_60d", "SLB_Schlumberger_ret_1d", "spx_abs_ret_max_5d", "HD_zscore_60d", "JNJ_ret_1d", "VOD_Vodafone_zscore_60d", "NFCI_ret_5d", "CPB_CampbellSoup_ret_20d", "IBEX_Spain_ret_20d", "3M_ret_5d", "EWS_Singapore_ret_5d", "heston_var_ev_h7", "ORCL_zscore_60d"], "is_new": true}, {"model_id": "new_h10_STRESS_LogisticRegression_N30_t1", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 10, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "DOW_Price_zscore_60d", "CPB_CampbellSoup_vol_20d", "AXP_Amex_vol_20d", "EMR_Emerson_ret_20d", "NWL_Newell_ret_20d", "XLV_Health_zscore_60d", "TED_Spread_zscore_60d", "XLF_Fin_vol_20d", "EWY_Korea_zscore_60d", "MO_AltriaMG_ret_1d", "3M_vol_20d", "MSTR_Bitcoin3_ret_20d", "QQQ_vol_20d", "MRK_Merck_zscore_60d", "INTC_ret_5d", "AORD_AUS_zscore_60d", "IYM_BasicMaterials_ret_20d", "PCAR_PaccarInc_ret_5d", "Nikkei_Japan_vol_20d", "TM_Telephone_vol_20d", "IWM_SmallCap_vol_20d", "EWS_Singapore_ret_5d", "GD_GeneralDynamics_zscore_60d", "HUM_Humana_ret_5d", "EWG_Germany_vol_20d", "HangSeng_HK_ret_1d", "MS_MorganStanley_ret_1d", "PPL_PPL_ret_1d"], "is_new": true}, {"model_id": "new_h10_STRESS_LogisticRegression_N30_t2", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 10, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EFFR_ret_1d", "LLY_zscore_60d", "WTI_Oil_FRED_zscore_60d", "MSTR_Bitcoin3_ret_1d", "heston_var_ev_h5", "EMR_Emerson_ret_20d", "Industrial_Production_zscore_60d", "MS_MorganStanley_ret_5d", "CMCSA_ret_1d", "HangSeng_HK_ret_5d", "PPL_PPL_ret_1d", "spx_abs_ret_max_5d", "SO_SouthernCo_ret_5d", "MS_MorganStanley_ret_1d", "DHR_vol_20d", "EXC_Exelon_ret_1d", "US3Y_Rate_ret_5d", "TGT_Target_zscore_60d", "ES_Evergy_ret_1d", "NVDA_vol_20d", "SBUX_zscore_60d", "SBUX_ret_5d", "MRK_Merck_zscore_60d", "EWH_HongKong_ret_5d", "EWC_Canada_zscore_60d", "XLF_Fin_vol_20d", "US5Y_Rate_ret_5d", "AMD_ret_5d"], "is_new": true}, {"model_id": "new_h10_STRESS_LogisticRegression_N30_t3", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 10, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "GILD_Gilead_ret_20d", "MS_MorganStanley_ret_5d", "GE_ret_1d", "SCHW_Schwab_ret_5d", "XLV_Health_zscore_60d", "Core_CPI_zscore_60d", "gjr_condvar_h1", "CPB_CampbellSoup_ret_5d", "heston_var_ev_h7", "HUM_Humana_ret_5d", "EFFR_ret_1d", "HD_ret_1d", "ORCL_vol_20d", "EWY_Korea_zscore_60d", "PLD_Prologis_ret_5d", "EXC_Exelon_ret_1d", "ASX_Australia_vol_20d", "BLK_BlackRock_zscore_60d", "AMGN_Amgen_ret_1d", "MO_AltriaMG_ret_1d", "NEE_NextEra_ret_20d", "heston_var_ev_h5", "Michigan_Sentiment_ret_20d", "CPB_CampbellSoup_zscore_60d", "TED_Spread_vol_20d", "HD_ret_5d", "MSTR_Bitcoin3_ret_1d", "heston_ev_h3"], "is_new": true}, {"model_id": "new_h10_STRESS_LogisticRegression_N30_t4", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 10, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "XOM_ret_20d", "BA_ret_1d", "PG_ret_20d", "T_ret_1d", "SO_SouthernCo_ret_5d", "US3Y_Rate_ret_5d", "NEE_NextEra_ret_20d", "AXP_Amex_ret_20d", "TED_Spread_zscore_60d", "MRK_Merck_zscore_60d", "EFFR_vol_20d", "EFFR_ret_1d", "EWM_Malaysia_ret_1d", "DAX_Germany_zscore_60d", "XLK_Tech_zscore_60d", "Nikkei_Japan_vol_20d", "EQR_Equity_ret_1d", "EWL_Switzerland_zscore_60d", "FedFunds_zscore_60d", "BDX_Becton_Dickinson_ret_20d", "CTAS_Cintas_vol_20d", "AMD_ret_5d", "EXC_Exelon_zscore_60d", "M_Macys_vol_20d", "ENB_EnbridgeInc_ret_1d", "EWG_Germany_vol_20d", "TM_Telephone_ret_1d", "SPY_zscore_60d"], "is_new": true}, {"model_id": "new_h10_STRESS_LogisticRegression_N30_t5", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 10, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "AVB_AvalonBay_zscore_60d", "XLB_Materials_zscore_60d", "HD_zscore_60d", "EWQ_France_ret_20d", "BTI_BritishAmerican_ret_20d", "AORD_AUS_zscore_60d", "MSTR_Bitcoin3_ret_5d", "ASX_Australia_ret_5d", "GILD_Gilead_ret_20d", "TM_Telephone_ret_1d", "Michigan_Sentiment_ret_20d", "US3M_Rate_vol_20d", "EWA_Australia_zscore_60d", "LLY_zscore_60d", "SBUX_ret_5d", "EQR_Equity_ret_1d", "EWQ_France_zscore_60d", "HangSeng_HK_vol_20d", "PFE_ret_1d", "US3Y_Rate_ret_5d", "M_Macys_vol_20d", "HD_ret_5d", "VRP_ma5", "US6M_Rate_ret_20d", "AMD_ret_5d", "3M_vol_20d", "FedFunds_zscore_60d", "SJM_JM_Smucker_ret_1d"], "is_new": true}, {"model_id": "new_h10_STRESS_LogisticRegression_N30_t6", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 10, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWL_Switzerland_vol_20d", "MS_MorganStanley_ret_5d", "US6M_Rate_ret_20d", "NWL_Newell_ret_20d", "Michigan_Sentiment_ret_20d", "DHR_ret_1d", "NEE_NextEra_ret_20d", "AMD_ret_1d", "VRP_ma5", "EWG_Germany_ret_20d", "EWA_Australia_ret_1d", "vix_mean_abs_ret_5d", "CPB_CampbellSoup_zscore_60d", "ORCL_zscore_60d", "gjr_condvar_h1", "spx_abs_ret_max_5d", "DAX_Germany_vol_20d", "ES_Evergy_ret_1d", "EWJ_Japan_vol_20d", "SBUX_vol_20d", "PAYX_Paychex_vol_20d", "SLB_Schlumberger_ret_5d", "SCHW_Schwab_ret_5d", "FedFunds_zscore_60d", "NVDA_vol_20d", "CI_Cigna_vol_20d", "TED_Spread_zscore_60d", "US7Y_Rate_ret_20d"], "is_new": true}, {"model_id": "new_h10_STRESS_LogisticRegression_N30_t7", "algo": "LogisticRegression", "regime": "STRESS", "horizon": 10, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "US7Y_Rate_ret_20d", "Industrial_Production_zscore_60d", "US1Y_Rate_ret_5d", "HD_zscore_60d", "EQR_Equity_ret_1d", "heston_ev_h3", "EWQ_France_zscore_60d", "T10Y2Y_Spread_ret_5d", "EOG_EOGResources_vol_20d", "EWG_Germany_ret_20d", "Nikkei_Japan_vol_20d", "vix_acceleration_1d", "Michigan_Sentiment_ret_20d", "SBUX_zscore_60d", "IWM_SmallCap_vol_20d", "HD_ret_20d", "EWQ_France_ret_20d", "3M_vol_20d", "AMGN_Amgen_ret_1d", "INTC_ret_1d", "EWC_Canada_zscore_60d", "MSTR_Bitcoin3_ret_20d", "TED_Spread_zscore_60d", "LOW_Lowes_ret_20d", "EXC_Exelon_ret_1d", "LUV_SouthwestAir_ret_5d", "CPB_CampbellSoup_ret_20d", "ES_Evergy_ret_1d"], "is_new": true}, {"model_id": "new_h10_GLOBAL_XGBoost_N5_t0", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 10, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "HD_zscore_60d", "SJM_JM_Smucker_ret_5d", "EWM_Malaysia_ret_1d"], "is_new": true}, {"model_id": "new_h10_GLOBAL_XGBoost_N5_t1", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 10, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "US3M_Rate_zscore_60d", "SBUX_ret_5d", "INTC_ret_1d"], "is_new": true}, {"model_id": "new_h10_GLOBAL_XGBoost_N5_t2", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 10, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "DIS_vol_20d", "CLX_Clorox_vol_20d", "EWA_Australia_zscore_60d"], "is_new": true}, {"model_id": "new_h10_GLOBAL_XGBoost_N5_t3", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 10, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWJ_Japan_vol_20d", "PAYX_Paychex_zscore_60d", "EOG_EOGResources_vol_20d"], "is_new": true}, {"model_id": "new_h10_GLOBAL_XGBoost_N5_t4", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 10, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "PLD_Prologis_ret_5d", "Brent_Oil_FRED_ret_5d", "IYM_BasicMaterials_ret_20d"], "is_new": true}, {"model_id": "new_h10_GLOBAL_XGBoost_N5_t5", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 10, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "gjr_condvar_h1", "EWS_Singapore_ret_5d", "HD_ret_1d"], "is_new": true}, {"model_id": "new_h10_GLOBAL_XGBoost_N5_t6", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 10, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "TXN_vol_20d", "DAX_Germany_vol_20d", "EWM_Malaysia_zscore_60d"], "is_new": true}, {"model_id": "new_h10_GLOBAL_XGBoost_N5_t7", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 10, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "INTC_ret_1d", "EFFR_ret_1d", "SO_SouthernCo_ret_5d"], "is_new": true}, {"model_id": "new_h10_GLOBAL_XGBoost_N8_t0", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 10, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "HD_zscore_60d", "heston_var_ev_h7", "AMZN_ret_5d", "CLX_Clorox_vol_20d", "QQQ_vol_20d", "EQIX_Equinix_ret_5d"], "is_new": true}, {"model_id": "new_h10_GLOBAL_XGBoost_N8_t1", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 10, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EOG_EOGResources_ret_5d", "PLD_Prologis_ret_5d", "EWS_Singapore_ret_5d", "CMCSA_ret_1d", "CTAS_Cintas_vol_20d", "ORCL_zscore_60d"], "is_new": true}, {"model_id": "new_h10_GLOBAL_XGBoost_N8_t2", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 10, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EFFR_ret_1d", "HD_zscore_60d", "EWM_Malaysia_vol_20d", "DHR_vol_20d", "HD_ret_1d", "HangSeng_HK_vol_20d"], "is_new": true}, {"model_id": "new_h10_GLOBAL_XGBoost_N8_t3", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 10, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "PAYX_Paychex_vol_20d", "heston_var_ev_h5", "Nikkei_Japan_vol_20d", "CTAS_Cintas_vol_20d", "ORCL_vol_20d", "CMCSA_ret_1d"], "is_new": true}, {"model_id": "new_h10_GLOBAL_XGBoost_N8_t4", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 10, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "spx_momentum_3d", "MRK_Merck_zscore_60d", "EOG_EOGResources_ret_5d", "CPB_CampbellSoup_vol_20d", "US6M_Rate_ret_20d", "SBUX_zscore_60d"], "is_new": true}, {"model_id": "new_h10_GLOBAL_XGBoost_N8_t5", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 10, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "PG_ret_20d", "heston_var_ev_h3", "Core_CPI_zscore_60d", "AMD_ret_5d", "MSTR_Bitcoin3_ret_20d", "US30Y_Rate_ret_20d"], "is_new": true}, {"model_id": "new_h10_GLOBAL_XGBoost_N8_t6", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 10, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "CPB_CampbellSoup_zscore_60d", "PG_ret_20d", "SO_SouthernCo_ret_5d", "XLY_Disc_vol_20d", "AMGN_Amgen_ret_1d", "PAYX_Paychex_ret_20d"], "is_new": true}, {"model_id": "new_h10_GLOBAL_XGBoost_N8_t7", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 10, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWY_Korea_zscore_60d", "heston_var_ev_h3", "GILD_Gilead_ret_20d", "IBEX_Spain_ret_20d", "SBUX_vol_20d", "NVDA_vol_20d"], "is_new": true}, {"model_id": "new_h10_GLOBAL_XGBoost_N10_t0", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 10, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "T10Y2Y_Spread_ret_5d", "CCI_CrownCastle_vol_20d", "AORD_AUS_zscore_60d", "DHR_ret_1d", "Nikkei_Japan_zscore_60d", "VVIX_ret_20d", "EOG_EOGResources_ret_5d", "spx_momentum_3d"], "is_new": true}, {"model_id": "new_h10_GLOBAL_XGBoost_N10_t1", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 10, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "AMZN_ret_5d", "EWQ_France_zscore_60d", "US3M_Rate_zscore_60d", "TGT_Target_zscore_60d", "DIS_vol_20d", "AMD_ret_5d", "AMD_ret_1d", "EQR_Equity_ret_1d"], "is_new": true}, {"model_id": "new_h10_GLOBAL_XGBoost_N10_t2", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 10, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "Brent_Oil_FRED_ret_5d", "EWQ_France_zscore_60d", "DE_Deere_ret_5d", "AVB_AvalonBay_zscore_60d", "HangSeng_HK_ret_1d", "PLD_Prologis_ret_5d", "hmm_p_stress", "CPB_CampbellSoup_ret_20d"], "is_new": true}, {"model_id": "new_h10_GLOBAL_XGBoost_N10_t3", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 10, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "WTI_Oil_FRED_zscore_60d", "EWM_Malaysia_vol_20d", "MS_MorganStanley_zscore_60d", "EWQ_France_zscore_60d", "NEE_NextEra_ret_20d", "HangSeng_HK_ret_5d", "PPL_PPL_ret_1d", "ORCL_vol_20d"], "is_new": true}, {"model_id": "new_h10_GLOBAL_XGBoost_N10_t4", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 10, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "M_Macys_vol_20d", "AXP_Amex_vol_20d", "EXC_Exelon_zscore_60d", "DE_Deere_vol_20d", "heston_var_ev_h5", "LMT_LockheedMartin_ret_1d", "MSTR_Bitcoin3_ret_5d", "BTI_BritishAmerican_ret_20d"], "is_new": true}, {"model_id": "new_h10_GLOBAL_XGBoost_N10_t5", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 10, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "CMCSA_ret_1d", "PFE_ret_1d", "NWL_Newell_ret_20d", "VOD_Vodafone_zscore_60d", "ORCL_zscore_60d", "AORD_AUS_zscore_60d", "EWG_Germany_ret_20d", "EWL_Switzerland_zscore_60d"], "is_new": true}, {"model_id": "new_h10_GLOBAL_XGBoost_N10_t6", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 10, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "HangSeng_HK_ret_1d", "AXP_Amex_ret_20d", "EFFR_vol_20d", "EWH_HongKong_ret_5d", "Retail_Sales_zscore_60d", "vix_mean_abs_ret_5d", "EWA_Australia_ret_1d", "US7Y_Rate_ret_20d"], "is_new": true}, {"model_id": "new_h10_GLOBAL_XGBoost_N10_t7", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 10, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "spx_momentum_3d", "TM_Telephone_vol_20d", "AMD_ret_5d", "CPB_CampbellSoup_ret_5d", "XLB_Materials_zscore_60d", "EXC_Exelon_ret_1d", "EFFR_ret_1d", "US1Y_Rate_ret_5d"], "is_new": true}, {"model_id": "new_h10_GLOBAL_XGBoost_N12_t0", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 10, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWM_Malaysia_ret_1d", "US1Y_Rate_ret_5d", "Core_CPI_zscore_60d", "MRK_Merck_zscore_60d", "EFFR_ret_1d", "DHR_ret_1d", "SPY_zscore_60d", "US1Y_Rate_ret_20d", "EMR_Emerson_ret_20d", "3M_vol_20d"], "is_new": true}, {"model_id": "new_h10_GLOBAL_XGBoost_N12_t1", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 10, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWG_Germany_vol_20d", "SJM_JM_Smucker_ret_5d", "BA_ret_1d", "DAX_Germany_vol_20d", "EWM_Malaysia_ret_1d", "Michigan_Sentiment_ret_20d", "T10Y2Y_Spread_ret_5d", "EXC_Exelon_ret_1d", "NVDA_vol_20d", "MRK_Merck_zscore_60d"], "is_new": true}, {"model_id": "new_h10_GLOBAL_XGBoost_N12_t2", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 10, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "MS_MorganStanley_zscore_60d", "XOM_ret_1d", "DE_Deere_vol_20d", "EOG_EOGResources_ret_5d", "PFE_ret_1d", "MSTR_Bitcoin3_ret_5d", "EXC_Exelon_ret_1d", "CLX_Clorox_vol_20d", "US5Y_Rate_ret_5d", "US3M_Rate_vol_20d"], "is_new": true}, {"model_id": "new_h10_GLOBAL_XGBoost_N12_t3", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 10, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "T10Y2Y_Spread_ret_5d", "Core_CPI_zscore_60d", "EWM_Malaysia_ret_1d", "ORCL_zscore_60d", "CPB_CampbellSoup_zscore_60d", "MS_MorganStanley_zscore_60d", "AMD_ret_5d", "US3Y_Rate_ret_5d", "Retail_Sales_zscore_60d", "EWM_Malaysia_vol_20d"], "is_new": true}, {"model_id": "new_h10_GLOBAL_XGBoost_N12_t4", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 10, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "Brent_Oil_FRED_ret_20d", "IBEX_Spain_ret_20d", "VRP_ma5", "MSTR_Bitcoin3_ret_20d", "AMD_ret_1d", "PLD_Prologis_ret_5d", "SO_SouthernCo_ret_5d", "EWG_Germany_vol_20d", "NOC_Northrop_ret_20d", "spx_vol_5d"], "is_new": true}, {"model_id": "new_h10_GLOBAL_XGBoost_N12_t5", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 10, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "SJM_JM_Smucker_ret_5d", "EQIX_Equinix_ret_5d", "CPB_CampbellSoup_ret_20d", "EXC_Exelon_ret_1d", "CPB_CampbellSoup_zscore_60d", "DIS_vol_20d", "Retail_Sales_zscore_60d", "IBEX_Spain_ret_20d", "XLB_Materials_zscore_60d", "DOW_Price_zscore_60d"], "is_new": true}, {"model_id": "new_h10_GLOBAL_XGBoost_N12_t6", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 10, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "TGT_Target_zscore_60d", "EWM_Malaysia_vol_20d", "AMT_AmericanTower_ret_1d", "IYR_US_REIT2_zscore_60d", "ASX_Australia_vol_20d", "GD_GeneralDynamics_zscore_60d", "LLY_zscore_60d", "NFCI_ret_5d", "M_Macys_vol_20d", "SBUX_zscore_60d"], "is_new": true}, {"model_id": "new_h10_GLOBAL_XGBoost_N12_t7", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 10, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "SCHW_Schwab_ret_5d", "AMD_ret_5d", "EWM_Malaysia_vol_20d", "BTI_BritishAmerican_ret_5d", "DE_Deere_ret_5d", "Retail_Sales_zscore_60d", "DHR_vol_20d", "QQQ_vol_20d", "CMCSA_ret_1d", "BA_ret_1d"], "is_new": true}, {"model_id": "new_h10_GLOBAL_XGBoost_N15_t0", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 10, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "PPL_PPL_ret_1d", "US1Y_Rate_ret_20d", "SCHW_Schwab_ret_5d", "DAX_Germany_vol_20d", "DIS_vol_20d", "LMT_LockheedMartin_vol_20d", "T10Y2Y_Spread_ret_5d", "AMD_ret_5d", "CLX_Clorox_vol_20d", "ASX_Australia_ret_5d", "BA_ret_1d", "PCAR_PaccarInc_ret_5d", "US3M_Rate_zscore_60d"], "is_new": true}, {"model_id": "new_h10_GLOBAL_XGBoost_N15_t1", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 10, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "MSTR_Bitcoin3_ret_20d", "PAYX_Paychex_ret_20d", "SLB_Schlumberger_ret_5d", "GILD_Gilead_ret_20d", "LOW_Lowes_ret_5d", "EWQ_France_ret_20d", "US1Y_Rate_ret_5d", "CPB_CampbellSoup_ret_5d", "CPB_CampbellSoup_vol_20d", "LUV_SouthwestAir_ret_5d", "US6M_Rate_ret_20d", "LMT_LockheedMartin_vol_20d", "PAYX_Paychex_vol_20d"], "is_new": true}, {"model_id": "new_h10_GLOBAL_XGBoost_N15_t2", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 10, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "AXP_Amex_ret_20d", "CPB_CampbellSoup_vol_20d", "VOD_Vodafone_zscore_60d", "Nikkei_Japan_vol_20d", "DIS_vol_20d", "AMGN_Amgen_ret_1d", "ORCL_vol_20d", "MRK_Merck_zscore_60d", "AMD_ret_5d", "WTI_Oil_FRED_zscore_60d", "TM_Telephone_ret_1d", "PPL_PPL_ret_1d", "CPB_CampbellSoup_ret_20d"], "is_new": true}, {"model_id": "new_h10_GLOBAL_XGBoost_N15_t3", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 10, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "AVB_AvalonBay_zscore_60d", "IYR_US_REIT2_zscore_60d", "FedFunds_zscore_60d", "EWA_Australia_zscore_60d", "DAX_Germany_vol_20d", "LMT_LockheedMartin_vol_20d", "3M_ret_5d", "MS_MorganStanley_ret_1d", "SO_SouthernCo_ret_5d", "XLB_Materials_zscore_60d", "HUM_Humana_ret_5d", "NFCI_ret_5d", "VOD_Vodafone_zscore_60d"], "is_new": true}, {"model_id": "new_h10_GLOBAL_XGBoost_N15_t4", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 10, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "CTAS_Cintas_vol_20d", "vix_mean_abs_ret_5d", "ORCL_vol_20d", "EWS_Singapore_ret_5d", "AXP_Amex_ret_20d", "NEE_NextEra_ret_20d", "DE_Deere_vol_20d", "PAYX_Paychex_vol_20d", "HangSeng_HK_ret_5d", "DOW_Price_zscore_60d", "XLY_Disc_vol_20d", "DIS_vol_20d", "DHR_vol_20d"], "is_new": true}, {"model_id": "new_h10_GLOBAL_XGBoost_N15_t5", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 10, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "XOM_ret_20d", "EWG_Germany_vol_20d", "NWL_Newell_ret_20d", "FedFunds_zscore_60d", "Industrial_Production_zscore_60d", "hmm_p_stress", "BTI_BritishAmerican_ret_5d", "DE_Deere_ret_5d", "ITT_ITTInc_ret_5d", "HangSeng_HK_ret_5d", "MS_MorganStanley_zscore_60d", "HangSeng_HK_ret_1d", "EWY_Korea_zscore_60d"], "is_new": true}, {"model_id": "new_h10_GLOBAL_XGBoost_N15_t6", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 10, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "HD_zscore_60d", "PAYX_Paychex_zscore_60d", "US1Y_Rate_ret_20d", "LMT_LockheedMartin_vol_20d", "EWS_Singapore_ret_5d", "ASX_Australia_ret_5d", "DHR_ret_1d", "AMD_ret_1d", "US5Y_Rate_ret_5d", "vix_mean_abs_ret_5d", "DE_Deere_vol_20d", "MS_MorganStanley_ret_5d", "HangSeng_HK_ret_5d"], "is_new": true}, {"model_id": "new_h10_GLOBAL_XGBoost_N15_t7", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 10, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "CCI_CrownCastle_vol_20d", "CPB_CampbellSoup_vol_20d", "3M_ret_5d", "HangSeng_HK_ret_5d", "NWL_Newell_ret_20d", "EXC_Exelon_ret_1d", "SBUX_vol_20d", "DHR_ret_1d", "AMZN_ret_5d", "EOG_EOGResources_ret_5d", "AMD_ret_1d", "SBUX_ret_5d", "VVIX_ret_20d"], "is_new": true}, {"model_id": "new_h10_GLOBAL_XGBoost_N20_t0", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 10, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "US3M_Rate_zscore_60d", "SPY_zscore_60d", "PCAR_PaccarInc_ret_5d", "EWL_Switzerland_vol_20d", "EQIX_Equinix_ret_5d", "EQR_Equity_ret_1d", "XLF_Fin_vol_20d", "DAX_Germany_vol_20d", "MS_MorganStanley_zscore_60d", "ITT_ITTInc_ret_5d", "DOW_Price_zscore_60d", "BDX_Becton_Dickinson_ret_20d", "AMZN_ret_5d", "WTI_Oil_FRED_zscore_60d", "ASX_Australia_vol_20d", "EXC_Exelon_zscore_60d", "XLB_Materials_zscore_60d", "BLK_BlackRock_zscore_60d"], "is_new": true}, {"model_id": "new_h10_GLOBAL_XGBoost_N20_t1", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 10, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "AMT_AmericanTower_ret_1d", "EWY_Korea_zscore_60d", "EFFR_ret_1d", "MS_MorganStanley_ret_1d", "GILD_Gilead_ret_20d", "Retail_Sales_zscore_60d", "BA_ret_1d", "MRK_Merck_zscore_60d", "NWL_Newell_ret_20d", "SLB_Schlumberger_ret_5d", "3M_vol_20d", "MS_MorganStanley_zscore_60d", "HD_ret_1d", "XOM_ret_20d", "AMGN_Amgen_ret_1d", "AVB_AvalonBay_zscore_60d", "EFFR_vol_20d", "hmm_p_stress"], "is_new": true}, {"model_id": "new_h10_GLOBAL_XGBoost_N20_t2", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 10, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWJ_Japan_vol_20d", "NOC_Northrop_ret_20d", "HD_ret_20d", "MO_AltriaMG_ret_1d", "LUV_SouthwestAir_ret_5d", "MS_MorganStanley_ret_1d", "EFFR_ret_1d", "LOW_Lowes_ret_20d", "EXC_Exelon_zscore_60d", "DE_Deere_vol_20d", "EMR_Emerson_ret_20d", "MS_MorganStanley_zscore_60d", "LMT_LockheedMartin_vol_20d", "AMZN_ret_5d", "NEE_NextEra_ret_20d", "EWL_Switzerland_zscore_60d", "ASX_Australia_vol_20d", "HangSeng_HK_ret_5d"], "is_new": true}, {"model_id": "new_h10_GLOBAL_XGBoost_N20_t3", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 10, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "hmm_p_stress", "EOG_EOGResources_vol_20d", "SPY_zscore_60d", "HD_zscore_60d", "US7Y_Rate_ret_20d", "HangSeng_HK_vol_20d", "JNJ_ret_1d", "EWL_Switzerland_vol_20d", "LOW_Lowes_ret_5d", "heston_var_ev_h5", "DE_Deere_vol_20d", "XOM_ret_20d", "IYM_BasicMaterials_ret_20d", "NEE_NextEra_ret_20d", "vix_acceleration_1d", "CPB_CampbellSoup_ret_5d", "ASX_Australia_ret_5d", "EWM_Malaysia_zscore_60d"], "is_new": true}, {"model_id": "new_h10_GLOBAL_XGBoost_N20_t4", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 10, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "SO_SouthernCo_ret_5d", "T_ret_1d", "PAYX_Paychex_ret_20d", "LLY_zscore_60d", "ASX_Australia_vol_20d", "spx_vol_5d", "TED_Spread_zscore_60d", "Nikkei_Japan_zscore_60d", "LOW_Lowes_ret_20d", "AVB_AvalonBay_zscore_60d", "NFCI_ret_5d", "EQIX_Equinix_ret_5d", "VOD_Vodafone_zscore_60d", "SLB_Schlumberger_ret_5d", "ITT_ITTInc_ret_5d", "spx_abs_ret_max_5d", "US6M_Rate_ret_20d", "DHR_ret_1d"], "is_new": true}, {"model_id": "new_h10_GLOBAL_XGBoost_N20_t5", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 10, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "AMD_ret_1d", "DE_Deere_vol_20d", "VRP_ma5", "XOM_ret_20d", "CTAS_Cintas_vol_20d", "3M_ret_5d", "MS_MorganStanley_ret_5d", "XLB_Materials_zscore_60d", "SBUX_vol_20d", "FedFunds_zscore_60d", "EWS_Singapore_ret_5d", "NEE_NextEra_ret_20d", "HD_zscore_60d", "US5Y_Rate_ret_5d", "MS_MorganStanley_ret_1d", "PG_ret_20d", "SBUX_zscore_60d", "EWG_Germany_vol_20d"], "is_new": true}, {"model_id": "new_h10_GLOBAL_XGBoost_N20_t6", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 10, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "DOW_Price_zscore_60d", "IBEX_Spain_ret_20d", "ASX_Australia_ret_5d", "TGT_Target_zscore_60d", "DIS_vol_20d", "Brent_Oil_FRED_ret_20d", "US1Y_Rate_ret_20d", "CI_Cigna_vol_20d", "EWA_Australia_zscore_60d", "Core_PCE_zscore_60d", "ITT_ITTInc_ret_5d", "CLX_Clorox_vol_20d", "LUV_SouthwestAir_ret_5d", "Core_CPI_zscore_60d", "US3M_Rate_zscore_60d", "GILD_Gilead_ret_20d", "SJM_JM_Smucker_ret_1d", "HangSeng_HK_vol_20d"], "is_new": true}, {"model_id": "new_h10_GLOBAL_XGBoost_N20_t7", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 10, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "AVB_AvalonBay_zscore_60d", "NEE_NextEra_ret_20d", "EWY_Korea_ret_20d", "US6M_Rate_ret_20d", "EWM_Malaysia_vol_20d", "VVIX_ret_20d", "XLV_Health_zscore_60d", "EOG_EOGResources_ret_5d", "AXP_Amex_vol_20d", "CLX_Clorox_vol_20d", "hmm_p_stress", "QQQ_vol_20d", "Retail_Sales_zscore_60d", "NFCI_ret_5d", "BLK_BlackRock_zscore_60d", "DIS_vol_20d", "EWH_HongKong_ret_5d", "FedFunds_zscore_60d"], "is_new": true}, {"model_id": "new_h10_GLOBAL_XGBoost_N25_t0", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 10, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "Industrial_Production_zscore_60d", "XLY_Disc_vol_20d", "GILD_Gilead_ret_20d", "LLY_zscore_60d", "BTI_BritishAmerican_ret_5d", "CI_Cigna_vol_20d", "EWM_Malaysia_vol_20d", "GE_ret_1d", "EWH_HongKong_ret_5d", "MSTR_Bitcoin3_ret_1d", "BA_ret_1d", "MSTR_Bitcoin3_ret_20d", "AMD_ret_1d", "SBUX_vol_20d", "PPL_PPL_ret_1d", "DAX_Germany_vol_20d", "EWG_Germany_vol_20d", "spx_abs_ret_max_5d", "heston_var_ev_h7", "CCI_CrownCastle_vol_20d", "EWY_Korea_zscore_60d", "EWG_Germany_ret_20d", "QQQ_vol_20d"], "is_new": true}, {"model_id": "new_h10_GLOBAL_XGBoost_N25_t1", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 10, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "Nikkei_Japan_zscore_60d", "EWM_Malaysia_ret_1d", "EWL_Switzerland_zscore_60d", "PAYX_Paychex_zscore_60d", "DHR_vol_20d", "IYM_BasicMaterials_ret_20d", "XLK_Tech_zscore_60d", "EOG_EOGResources_vol_20d", "MS_MorganStanley_zscore_60d", "Nikkei_Japan_vol_20d", "PCAR_PaccarInc_ret_5d", "LMT_LockheedMartin_vol_20d", "3M_vol_20d", "EWG_Germany_vol_20d", "CI_Cigna_vol_20d", "IWM_SmallCap_vol_20d", "CCI_CrownCastle_vol_20d", "AMD_ret_1d", "spx_vol_5d", "ASX_Australia_ret_5d", "3M_ret_5d", "SBUX_zscore_60d", "NFCI_ret_5d"], "is_new": true}, {"model_id": "new_h10_GLOBAL_XGBoost_N25_t2", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 10, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "AMGN_Amgen_ret_1d", "PG_ret_20d", "EWA_Australia_zscore_60d", "DHR_vol_20d", "SJM_JM_Smucker_ret_1d", "EOG_EOGResources_vol_20d", "EWQ_France_ret_20d", "GE_ret_1d", "DAX_Germany_zscore_60d", "AMZN_ret_5d", "EWG_Germany_vol_20d", "AMD_ret_1d", "XLY_Disc_vol_20d", "Industrial_Production_zscore_60d", "CCI_CrownCastle_vol_20d", "DE_Deere_vol_20d", "HD_ret_20d", "vix_acceleration_1d", "EXC_Exelon_ret_1d", "CLX_Clorox_vol_20d", "ES_Evergy_ret_1d", "SBUX_ret_5d", "XOM_ret_1d"], "is_new": true}, {"model_id": "new_h10_GLOBAL_XGBoost_N25_t3", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 10, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "MRK_Merck_zscore_60d", "XOM_ret_20d", "GILD_Gilead_ret_20d", "EQIX_Equinix_ret_5d", "Michigan_Sentiment_ret_20d", "VOD_Vodafone_zscore_60d", "XLV_Health_zscore_60d", "US7Y_Rate_ret_20d", "SJM_JM_Smucker_ret_5d", "EWY_Korea_zscore_60d", "Industrial_Production_zscore_60d", "3M_ret_5d", "US30Y_Rate_ret_20d", "EWJ_Japan_vol_20d", "PLD_Prologis_ret_5d", "EWY_Korea_ret_20d", "PAYX_Paychex_ret_20d", "NWL_Newell_ret_20d", "US1Y_Rate_ret_20d", "US3M_Rate_vol_20d", "LMT_LockheedMartin_ret_1d", "Nikkei_Japan_vol_20d", "AMT_AmericanTower_ret_1d"], "is_new": true}, {"model_id": "new_h10_GLOBAL_XGBoost_N25_t4", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 10, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "FedFunds_zscore_60d", "AVB_AvalonBay_zscore_60d", "CPB_CampbellSoup_zscore_60d", "AMT_AmericanTower_ret_1d", "BDX_Becton_Dickinson_ret_20d", "GILD_Gilead_ret_20d", "EQR_Equity_ret_1d", "ENB_EnbridgeInc_ret_1d", "3M_ret_5d", "DHR_vol_20d", "Core_PCE_zscore_60d", "Industrial_Production_zscore_60d", "SO_SouthernCo_ret_5d", "MRK_Merck_zscore_60d", "EWQ_France_zscore_60d", "NFCI_ret_5d", "IYR_US_REIT2_zscore_60d", "EXC_Exelon_zscore_60d", "EWY_Korea_zscore_60d", "EWH_HongKong_ret_5d", "IYM_BasicMaterials_ret_20d", "US3Y_Rate_ret_5d", "EFFR_vol_20d"], "is_new": true}, {"model_id": "new_h10_GLOBAL_XGBoost_N25_t5", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 10, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "XOM_ret_1d", "ORCL_zscore_60d", "NVDA_vol_20d", "HUM_Humana_ret_5d", "DHR_vol_20d", "DHR_ret_1d", "EWL_Switzerland_zscore_60d", "GILD_Gilead_ret_20d", "CPB_CampbellSoup_vol_20d", "SBUX_zscore_60d", "EXC_Exelon_zscore_60d", "EWA_Australia_zscore_60d", "US3M_Rate_vol_20d", "PFE_ret_1d", "Core_PCE_zscore_60d", "XLV_Health_zscore_60d", "ORCL_vol_20d", "Brent_Oil_FRED_ret_20d", "heston_var_ev_h3", "HangSeng_HK_ret_1d", "EWC_Canada_zscore_60d", "EWG_Germany_vol_20d", "EWS_Singapore_ret_5d"], "is_new": true}, {"model_id": "new_h10_GLOBAL_XGBoost_N25_t6", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 10, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "HD_ret_20d", "EWJ_Japan_vol_20d", "US7Y_Rate_ret_20d", "SLB_Schlumberger_ret_5d", "EWL_Switzerland_vol_20d", "XOM_ret_20d", "spx_momentum_3d", "T_ret_1d", "ITT_ITTInc_ret_5d", "EWC_Canada_zscore_60d", "HUM_Humana_ret_5d", "DOW_Price_zscore_60d", "NVDA_vol_20d", "EWA_Australia_ret_1d", "ASX_Australia_ret_5d", "CMCSA_ret_1d", "LOW_Lowes_ret_5d", "PAYX_Paychex_ret_20d", "PLD_Prologis_ret_5d", "US6M_Rate_ret_20d", "GE_ret_1d", "AORD_AUS_zscore_60d", "US3Y_Rate_ret_5d"], "is_new": true}, {"model_id": "new_h10_GLOBAL_XGBoost_N25_t7", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 10, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "VVIX_ret_20d", "AMGN_Amgen_ret_1d", "EWL_Switzerland_zscore_60d", "heston_var_ev_h5", "PFE_ret_1d", "BTI_BritishAmerican_ret_5d", "EWJ_Japan_vol_20d", "US6M_Rate_ret_20d", "3M_ret_5d", "PPL_PPL_ret_1d", "MSTR_Bitcoin3_ret_1d", "SJM_JM_Smucker_ret_5d", "EWY_Korea_ret_20d", "TXN_vol_20d", "HangSeng_HK_vol_20d", "CPB_CampbellSoup_ret_5d", "DE_Deere_vol_20d", "EWA_Australia_zscore_60d", "Michigan_Sentiment_ret_20d", "CTAS_Cintas_vol_20d", "CCI_CrownCastle_vol_20d", "Core_CPI_zscore_60d", "EWH_HongKong_ret_5d"], "is_new": true}, {"model_id": "new_h10_GLOBAL_XGBoost_N30_t0", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 10, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "MS_MorganStanley_ret_5d", "Brent_Oil_FRED_ret_20d", "XLK_Tech_zscore_60d", "AXP_Amex_ret_20d", "CPB_CampbellSoup_ret_20d", "MO_AltriaMG_ret_1d", "LLY_zscore_60d", "DE_Deere_vol_20d", "IYR_US_REIT2_zscore_60d", "NOC_Northrop_ret_20d", "HangSeng_HK_vol_20d", "WTI_Oil_FRED_zscore_60d", "TXN_vol_20d", "PPL_PPL_ret_1d", "ORCL_vol_20d", "Retail_Sales_zscore_60d", "HangSeng_HK_ret_1d", "SO_SouthernCo_ret_5d", "XLB_Materials_zscore_60d", "LOW_Lowes_ret_20d", "GD_GeneralDynamics_zscore_60d", "EWY_Korea_zscore_60d", "DOW_Price_zscore_60d", "Brent_Oil_FRED_ret_5d", "EWG_Germany_ret_20d", "BTI_BritishAmerican_ret_5d", "PCAR_PaccarInc_ret_5d", "BA_ret_1d"], "is_new": true}, {"model_id": "new_h10_GLOBAL_XGBoost_N30_t1", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 10, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "Nikkei_Japan_zscore_60d", "HangSeng_HK_ret_5d", "LOW_Lowes_ret_20d", "SBUX_vol_20d", "PAYX_Paychex_vol_20d", "MSTR_Bitcoin3_ret_5d", "EFFR_ret_1d", "EWM_Malaysia_zscore_60d", "BDX_Becton_Dickinson_ret_20d", "MRK_Merck_zscore_60d", "heston_var_ev_h3", "EWA_Australia_ret_1d", "BLK_BlackRock_zscore_60d", "IYR_US_REIT2_zscore_60d", "CTAS_Cintas_vol_20d", "BA_ret_1d", "gjr_condvar_h1", "AVB_AvalonBay_zscore_60d", "Brent_Oil_FRED_ret_5d", "AMD_ret_5d", "EOG_EOGResources_ret_5d", "INTC_ret_5d", "EWG_Germany_vol_20d", "EQIX_Equinix_ret_5d", "spx_vol_5d", "TM_Telephone_ret_1d", "CPB_CampbellSoup_ret_5d", "EWQ_France_ret_20d"], "is_new": true}, {"model_id": "new_h10_GLOBAL_XGBoost_N30_t2", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 10, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "CPB_CampbellSoup_ret_5d", "PPL_PPL_ret_1d", "SCHW_Schwab_ret_5d", "NVDA_vol_20d", "EWL_Switzerland_zscore_60d", "EWQ_France_zscore_60d", "CMCSA_ret_1d", "BLK_BlackRock_zscore_60d", "XOM_ret_20d", "PFE_ret_1d", "DAX_Germany_vol_20d", "HD_zscore_60d", "MS_MorganStanley_zscore_60d", "SLB_Schlumberger_ret_5d", "EWC_Canada_zscore_60d", "SJM_JM_Smucker_ret_5d", "T_ret_1d", "heston_var_ev_h7", "US5Y_Rate_ret_5d", "AMD_ret_1d", "DE_Deere_vol_20d", "LOW_Lowes_ret_20d", "HD_ret_5d", "US7Y_Rate_ret_20d", "XLF_Fin_vol_20d", "EWL_Switzerland_vol_20d", "AXP_Amex_vol_20d", "MSTR_Bitcoin3_ret_5d"], "is_new": true}, {"model_id": "new_h10_GLOBAL_XGBoost_N30_t3", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 10, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "spx_momentum_3d", "XLB_Materials_zscore_60d", "ENB_EnbridgeInc_ret_1d", "EOG_EOGResources_vol_20d", "EWM_Malaysia_zscore_60d", "ASX_Australia_vol_20d", "Nikkei_Japan_zscore_60d", "MO_AltriaMG_ret_1d", "BTI_BritishAmerican_ret_20d", "Retail_Sales_zscore_60d", "PFE_ret_1d", "US3M_Rate_vol_20d", "EQIX_Equinix_ret_5d", "HUM_Humana_ret_5d", "CPB_CampbellSoup_ret_20d", "CLX_Clorox_vol_20d", "SBUX_zscore_60d", "AXP_Amex_vol_20d", "EWL_Switzerland_zscore_60d", "T10Y2Y_Spread_ret_5d", "HangSeng_HK_ret_5d", "TM_Telephone_vol_20d", "NWL_Newell_ret_20d", "SO_SouthernCo_ret_5d", "heston_var_ev_h3", "SBUX_vol_20d", "EXC_Exelon_zscore_60d", "XLF_Fin_vol_20d"], "is_new": true}, {"model_id": "new_h10_GLOBAL_XGBoost_N30_t4", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 10, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "ES_Evergy_ret_1d", "ITT_ITTInc_ret_5d", "DE_Deere_vol_20d", "BLK_BlackRock_zscore_60d", "EWY_Korea_ret_20d", "CPB_CampbellSoup_ret_5d", "US3M_Rate_zscore_60d", "EWM_Malaysia_vol_20d", "EXC_Exelon_ret_1d", "Brent_Oil_FRED_ret_20d", "spx_abs_ret_max_5d", "LMT_LockheedMartin_vol_20d", "SPY_zscore_60d", "Core_CPI_zscore_60d", "TGT_Target_zscore_60d", "TED_Spread_zscore_60d", "AVB_AvalonBay_zscore_60d", "XLY_Disc_vol_20d", "PFE_ret_1d", "MSTR_Bitcoin3_ret_20d", "FedFunds_zscore_60d", "SJM_JM_Smucker_ret_1d", "CTAS_Cintas_vol_20d", "PLD_Prologis_ret_5d", "HD_zscore_60d", "EXC_Exelon_zscore_60d", "BA_ret_1d", "Industrial_Production_zscore_60d"], "is_new": true}, {"model_id": "new_h10_GLOBAL_XGBoost_N30_t5", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 10, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWG_Germany_vol_20d", "MO_AltriaMG_ret_1d", "hmm_p_stress", "INTC_ret_1d", "vix_acceleration_1d", "EMR_Emerson_ret_20d", "DE_Deere_ret_5d", "EWM_Malaysia_zscore_60d", "ES_Evergy_ret_1d", "ORCL_zscore_60d", "HangSeng_HK_vol_20d", "heston_var_ev_h7", "LUV_SouthwestAir_ret_5d", "EWC_Canada_zscore_60d", "EOG_EOGResources_ret_5d", "HD_ret_20d", "DHR_vol_20d", "EOG_EOGResources_vol_20d", "EWS_Singapore_ret_5d", "DAX_Germany_zscore_60d", "AXP_Amex_vol_20d", "NEE_NextEra_ret_20d", "BA_ret_1d", "M_Macys_vol_20d", "TED_Spread_zscore_60d", "Nikkei_Japan_vol_20d", "3M_vol_20d", "BDX_Becton_Dickinson_ret_20d"], "is_new": true}, {"model_id": "new_h10_GLOBAL_XGBoost_N30_t6", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 10, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWA_Australia_ret_1d", "AMD_ret_5d", "CPB_CampbellSoup_zscore_60d", "T_ret_1d", "Michigan_Sentiment_ret_20d", "heston_var_ev_h7", "DHR_ret_1d", "AMD_ret_1d", "EMR_Emerson_ret_20d", "TED_Spread_zscore_60d", "HUM_Humana_ret_5d", "Brent_Oil_FRED_ret_5d", "TM_Telephone_ret_1d", "LOW_Lowes_ret_20d", "EWY_Korea_zscore_60d", "DAX_Germany_zscore_60d", "SPY_zscore_60d", "DAX_Germany_vol_20d", "CLX_Clorox_vol_20d", "US1Y_Rate_ret_5d", "TGT_Target_zscore_60d", "PAYX_Paychex_vol_20d", "hmm_p_stress", "NEE_NextEra_ret_20d", "EOG_EOGResources_ret_5d", "QQQ_vol_20d", "MS_MorganStanley_ret_5d", "PCAR_PaccarInc_ret_5d"], "is_new": true}, {"model_id": "new_h10_GLOBAL_XGBoost_N30_t7", "algo": "XGBoost", "regime": "GLOBAL", "horizon": 10, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "AXP_Amex_vol_20d", "SPY_zscore_60d", "TGT_Target_zscore_60d", "EWA_Australia_ret_1d", "Michigan_Sentiment_ret_20d", "US6M_Rate_ret_20d", "INTC_ret_1d", "MSTR_Bitcoin3_ret_5d", "EWL_Switzerland_zscore_60d", "EWM_Malaysia_ret_1d", "QQQ_vol_20d", "CTAS_Cintas_vol_20d", "GILD_Gilead_ret_20d", "HangSeng_HK_ret_1d", "XOM_ret_20d", "CI_Cigna_vol_20d", "SLB_Schlumberger_ret_5d", "US3M_Rate_vol_20d", "IWM_SmallCap_vol_20d", "NWL_Newell_ret_20d", "LUV_SouthwestAir_ret_5d", "EWQ_France_zscore_60d", "AORD_AUS_zscore_60d", "3M_ret_5d", "MO_AltriaMG_ret_1d", "EQIX_Equinix_ret_5d", "EWL_Switzerland_vol_20d", "VOD_Vodafone_zscore_60d"], "is_new": true}, {"model_id": "new_h10_GLOBAL_LightGBM_N5_t0", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 10, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "SPY_zscore_60d", "XOM_ret_1d", "vix_mean_abs_ret_5d"], "is_new": true}, {"model_id": "new_h10_GLOBAL_LightGBM_N5_t1", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 10, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "TGT_Target_zscore_60d", "CTAS_Cintas_vol_20d", "GD_GeneralDynamics_zscore_60d"], "is_new": true}, {"model_id": "new_h10_GLOBAL_LightGBM_N5_t2", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 10, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "HD_ret_20d", "LOW_Lowes_ret_20d", "AXP_Amex_ret_20d"], "is_new": true}, {"model_id": "new_h10_GLOBAL_LightGBM_N5_t3", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 10, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "XLF_Fin_vol_20d", "PCAR_PaccarInc_ret_5d", "SBUX_zscore_60d"], "is_new": true}, {"model_id": "new_h10_GLOBAL_LightGBM_N5_t4", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 10, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "SBUX_zscore_60d", "DAX_Germany_vol_20d", "HangSeng_HK_vol_20d"], "is_new": true}, {"model_id": "new_h10_GLOBAL_LightGBM_N5_t5", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 10, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "HD_ret_1d", "EXC_Exelon_zscore_60d", "VRP_ma5"], "is_new": true}, {"model_id": "new_h10_GLOBAL_LightGBM_N5_t6", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 10, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "ENB_EnbridgeInc_ret_1d", "EWG_Germany_vol_20d", "AMGN_Amgen_ret_1d"], "is_new": true}, {"model_id": "new_h10_GLOBAL_LightGBM_N5_t7", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 10, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "LOW_Lowes_ret_20d", "XOM_ret_1d", "HangSeng_HK_ret_1d"], "is_new": true}, {"model_id": "new_h10_GLOBAL_LightGBM_N8_t0", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 10, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWH_HongKong_ret_5d", "XLF_Fin_vol_20d", "LOW_Lowes_ret_5d", "HD_ret_5d", "BDX_Becton_Dickinson_ret_20d", "US3M_Rate_zscore_60d"], "is_new": true}, {"model_id": "new_h10_GLOBAL_LightGBM_N8_t1", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 10, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "JNJ_ret_1d", "Core_PCE_zscore_60d", "FedFunds_zscore_60d", "CPB_CampbellSoup_ret_5d", "PAYX_Paychex_vol_20d", "Brent_Oil_FRED_ret_5d"], "is_new": true}, {"model_id": "new_h10_GLOBAL_LightGBM_N8_t2", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 10, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "IBEX_Spain_ret_20d", "XLF_Fin_vol_20d", "BLK_BlackRock_zscore_60d", "MS_MorganStanley_ret_1d", "WTI_Oil_FRED_zscore_60d", "AXP_Amex_vol_20d"], "is_new": true}, {"model_id": "new_h10_GLOBAL_LightGBM_N8_t3", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 10, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "GD_GeneralDynamics_zscore_60d", "ENB_EnbridgeInc_ret_1d", "GILD_Gilead_ret_20d", "IWM_SmallCap_vol_20d", "WTI_Oil_FRED_zscore_60d", "SJM_JM_Smucker_ret_5d"], "is_new": true}, {"model_id": "new_h10_GLOBAL_LightGBM_N8_t4", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 10, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "AMT_AmericanTower_ret_1d", "M_Macys_vol_20d", "EWG_Germany_ret_20d", "CPB_CampbellSoup_zscore_60d", "LMT_LockheedMartin_ret_1d", "US1Y_Rate_ret_20d"], "is_new": true}, {"model_id": "new_h10_GLOBAL_LightGBM_N8_t5", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 10, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "ENB_EnbridgeInc_ret_1d", "EWS_Singapore_ret_5d", "EWL_Switzerland_vol_20d", "CI_Cigna_vol_20d", "US6M_Rate_ret_20d", "CCI_CrownCastle_vol_20d"], "is_new": true}, {"model_id": "new_h10_GLOBAL_LightGBM_N8_t6", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 10, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "SBUX_ret_5d", "SJM_JM_Smucker_ret_1d", "MS_MorganStanley_ret_5d", "US3M_Rate_vol_20d", "SO_SouthernCo_ret_5d", "EOG_EOGResources_vol_20d"], "is_new": true}, {"model_id": "new_h10_GLOBAL_LightGBM_N8_t7", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 10, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "MS_MorganStanley_ret_1d", "INTC_ret_1d", "heston_var_ev_h5", "HD_ret_5d", "ENB_EnbridgeInc_ret_1d", "Core_CPI_zscore_60d"], "is_new": true}, {"model_id": "new_h10_GLOBAL_LightGBM_N10_t0", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 10, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWQ_France_zscore_60d", "EWL_Switzerland_vol_20d", "TGT_Target_zscore_60d", "AMZN_ret_5d", "EWG_Germany_ret_20d", "WTI_Oil_FRED_zscore_60d", "EOG_EOGResources_ret_5d", "FedFunds_zscore_60d"], "is_new": true}, {"model_id": "new_h10_GLOBAL_LightGBM_N10_t1", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 10, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "HangSeng_HK_vol_20d", "heston_var_ev_h5", "NFCI_ret_5d", "MS_MorganStanley_ret_5d", "DOW_Price_zscore_60d", "XOM_ret_1d", "US6M_Rate_ret_20d", "US7Y_Rate_ret_20d"], "is_new": true}, {"model_id": "new_h10_GLOBAL_LightGBM_N10_t2", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 10, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "NEE_NextEra_ret_20d", "gjr_condvar_h1", "EFFR_vol_20d", "EOG_EOGResources_vol_20d", "EXC_Exelon_zscore_60d", "LMT_LockheedMartin_ret_1d", "SPY_zscore_60d", "3M_vol_20d"], "is_new": true}, {"model_id": "new_h10_GLOBAL_LightGBM_N10_t3", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 10, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "TGT_Target_zscore_60d", "HD_ret_1d", "AMT_AmericanTower_ret_1d", "heston_ev_h3", "vix_acceleration_1d", "TM_Telephone_ret_1d", "SJM_JM_Smucker_ret_5d", "HUM_Humana_ret_5d"], "is_new": true}, {"model_id": "new_h10_GLOBAL_LightGBM_N10_t4", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 10, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "AXP_Amex_ret_20d", "XLB_Materials_zscore_60d", "heston_ev_h3", "VVIX_ret_20d", "EQIX_Equinix_ret_5d", "VRP_ma5", "IWM_SmallCap_vol_20d", "WTI_Oil_FRED_zscore_60d"], "is_new": true}, {"model_id": "new_h10_GLOBAL_LightGBM_N10_t5", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 10, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "US7Y_Rate_ret_20d", "DIS_vol_20d", "EFFR_ret_1d", "SBUX_ret_5d", "ASX_Australia_ret_5d", "XOM_ret_20d", "AMGN_Amgen_ret_1d", "T_ret_1d"], "is_new": true}, {"model_id": "new_h10_GLOBAL_LightGBM_N10_t6", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 10, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "LLY_zscore_60d", "GE_ret_1d", "3M_ret_5d", "QQQ_vol_20d", "LUV_SouthwestAir_ret_5d", "PAYX_Paychex_vol_20d", "Nikkei_Japan_vol_20d", "ES_Evergy_ret_1d"], "is_new": true}, {"model_id": "new_h10_GLOBAL_LightGBM_N10_t7", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 10, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EOG_EOGResources_ret_5d", "TXN_vol_20d", "Nikkei_Japan_zscore_60d", "VOD_Vodafone_zscore_60d", "DHR_ret_1d", "EQIX_Equinix_ret_5d", "EFFR_vol_20d", "AORD_AUS_zscore_60d"], "is_new": true}, {"model_id": "new_h10_GLOBAL_LightGBM_N12_t0", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 10, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "IWM_SmallCap_vol_20d", "SBUX_vol_20d", "Retail_Sales_zscore_60d", "US1Y_Rate_ret_20d", "US6M_Rate_ret_20d", "TM_Telephone_vol_20d", "EQR_Equity_ret_1d", "NOC_Northrop_ret_20d", "gjr_condvar_h1", "PG_ret_20d"], "is_new": true}, {"model_id": "new_h10_GLOBAL_LightGBM_N12_t1", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 10, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "AVB_AvalonBay_zscore_60d", "NWL_Newell_ret_20d", "TED_Spread_zscore_60d", "Brent_Oil_FRED_ret_20d", "SCHW_Schwab_ret_5d", "EWY_Korea_zscore_60d", "AORD_AUS_zscore_60d", "MSTR_Bitcoin3_ret_5d", "EWM_Malaysia_vol_20d", "TM_Telephone_ret_1d"], "is_new": true}, {"model_id": "new_h10_GLOBAL_LightGBM_N12_t2", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 10, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "INTC_ret_1d", "EOG_EOGResources_vol_20d", "HangSeng_HK_ret_5d", "CPB_CampbellSoup_zscore_60d", "GILD_Gilead_ret_20d", "CPB_CampbellSoup_ret_5d", "heston_var_ev_h5", "BTI_BritishAmerican_ret_5d", "PG_ret_20d", "ORCL_zscore_60d"], "is_new": true}, {"model_id": "new_h10_GLOBAL_LightGBM_N12_t3", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 10, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWQ_France_zscore_60d", "NVDA_vol_20d", "DAX_Germany_vol_20d", "TXN_vol_20d", "ITT_ITTInc_ret_5d", "MSTR_Bitcoin3_ret_5d", "EQIX_Equinix_ret_5d", "AMD_ret_5d", "CMCSA_ret_1d", "TM_Telephone_ret_1d"], "is_new": true}, {"model_id": "new_h10_GLOBAL_LightGBM_N12_t4", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 10, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "heston_ev_h3", "INTC_ret_5d", "Core_CPI_zscore_60d", "MSTR_Bitcoin3_ret_5d", "PPL_PPL_ret_1d", "DE_Deere_ret_5d", "MSTR_Bitcoin3_ret_20d", "DHR_ret_1d", "NOC_Northrop_ret_20d", "LMT_LockheedMartin_vol_20d"], "is_new": true}, {"model_id": "new_h10_GLOBAL_LightGBM_N12_t5", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 10, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWG_Germany_ret_20d", "INTC_ret_1d", "BA_ret_1d", "EQIX_Equinix_ret_5d", "EWM_Malaysia_ret_1d", "DIS_vol_20d", "EWJ_Japan_vol_20d", "BTI_BritishAmerican_ret_20d", "EXC_Exelon_zscore_60d", "EFFR_vol_20d"], "is_new": true}, {"model_id": "new_h10_GLOBAL_LightGBM_N12_t6", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 10, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "TGT_Target_zscore_60d", "AVB_AvalonBay_zscore_60d", "heston_var_ev_h5", "spx_vol_5d", "BTI_BritishAmerican_ret_5d", "GD_GeneralDynamics_zscore_60d", "XLV_Health_zscore_60d", "Michigan_Sentiment_ret_20d", "ASX_Australia_vol_20d", "VOD_Vodafone_zscore_60d"], "is_new": true}, {"model_id": "new_h10_GLOBAL_LightGBM_N12_t7", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 10, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "T10Y2Y_Spread_ret_5d", "LOW_Lowes_ret_20d", "EWL_Switzerland_vol_20d", "SPY_zscore_60d", "AMD_ret_5d", "INTC_ret_5d", "spx_vol_5d", "PFE_ret_1d", "XLY_Disc_vol_20d", "HD_ret_1d"], "is_new": true}, {"model_id": "new_h10_GLOBAL_LightGBM_N15_t0", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 10, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "DE_Deere_ret_5d", "US30Y_Rate_ret_20d", "EWL_Switzerland_zscore_60d", "GD_GeneralDynamics_zscore_60d", "US1Y_Rate_ret_5d", "HD_ret_20d", "CTAS_Cintas_vol_20d", "3M_vol_20d", "US6M_Rate_ret_20d", "EWG_Germany_ret_20d", "Nikkei_Japan_vol_20d", "Brent_Oil_FRED_ret_5d", "TM_Telephone_ret_1d"], "is_new": true}, {"model_id": "new_h10_GLOBAL_LightGBM_N15_t1", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 10, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "TM_Telephone_ret_1d", "LUV_SouthwestAir_ret_5d", "NVDA_vol_20d", "JNJ_ret_1d", "EXC_Exelon_ret_1d", "LOW_Lowes_ret_20d", "EWJ_Japan_vol_20d", "IBEX_Spain_ret_20d", "EFFR_vol_20d", "NWL_Newell_ret_20d", "EFFR_ret_1d", "hmm_p_stress", "MO_AltriaMG_ret_1d"], "is_new": true}, {"model_id": "new_h10_GLOBAL_LightGBM_N15_t2", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 10, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "MSTR_Bitcoin3_ret_20d", "EFFR_ret_1d", "spx_abs_ret_max_5d", "TXN_vol_20d", "GE_ret_1d", "HUM_Humana_ret_5d", "MSTR_Bitcoin3_ret_1d", "EWH_HongKong_ret_5d", "Brent_Oil_FRED_ret_5d", "DE_Deere_vol_20d", "gjr_condvar_h1", "CPB_CampbellSoup_ret_5d", "CMCSA_ret_1d"], "is_new": true}, {"model_id": "new_h10_GLOBAL_LightGBM_N15_t3", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 10, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "DAX_Germany_vol_20d", "GE_ret_1d", "DIS_vol_20d", "SJM_JM_Smucker_ret_1d", "PPL_PPL_ret_1d", "EWA_Australia_zscore_60d", "HUM_Humana_ret_5d", "VOD_Vodafone_zscore_60d", "CPB_CampbellSoup_ret_20d", "3M_vol_20d", "JNJ_ret_1d", "EWY_Korea_ret_20d", "HangSeng_HK_ret_5d"], "is_new": true}, {"model_id": "new_h10_GLOBAL_LightGBM_N15_t4", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 10, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "US3Y_Rate_ret_5d", "US6M_Rate_ret_20d", "PCAR_PaccarInc_ret_5d", "DE_Deere_vol_20d", "Nikkei_Japan_zscore_60d", "PLD_Prologis_ret_5d", "VOD_Vodafone_zscore_60d", "EWG_Germany_ret_20d", "HD_ret_5d", "NVDA_vol_20d", "T_ret_1d", "CI_Cigna_vol_20d", "HD_ret_1d"], "is_new": true}, {"model_id": "new_h10_GLOBAL_LightGBM_N15_t5", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 10, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "AXP_Amex_ret_20d", "AMT_AmericanTower_ret_1d", "EFFR_ret_1d", "EQR_Equity_ret_1d", "XLK_Tech_zscore_60d", "TGT_Target_zscore_60d", "EWM_Malaysia_vol_20d", "VRP_ma5", "Michigan_Sentiment_ret_20d", "AMZN_ret_5d", "MS_MorganStanley_zscore_60d", "JNJ_ret_1d", "DAX_Germany_vol_20d"], "is_new": true}, {"model_id": "new_h10_GLOBAL_LightGBM_N15_t6", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 10, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "SPY_zscore_60d", "EQR_Equity_ret_1d", "ES_Evergy_ret_1d", "XOM_ret_20d", "GE_ret_1d", "GILD_Gilead_ret_20d", "WTI_Oil_FRED_zscore_60d", "DAX_Germany_zscore_60d", "HD_ret_1d", "IBEX_Spain_ret_20d", "EQIX_Equinix_ret_5d", "HD_ret_5d", "heston_var_ev_h3"], "is_new": true}, {"model_id": "new_h10_GLOBAL_LightGBM_N15_t7", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 10, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWL_Switzerland_zscore_60d", "Nikkei_Japan_zscore_60d", "EWY_Korea_ret_20d", "QQQ_vol_20d", "heston_var_ev_h3", "ITT_ITTInc_ret_5d", "NFCI_ret_5d", "EMR_Emerson_ret_20d", "NEE_NextEra_ret_20d", "hmm_p_stress", "Brent_Oil_FRED_ret_5d", "EQR_Equity_ret_1d", "CTAS_Cintas_vol_20d"], "is_new": true}, {"model_id": "new_h10_GLOBAL_LightGBM_N20_t0", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 10, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "CI_Cigna_vol_20d", "SBUX_zscore_60d", "TM_Telephone_vol_20d", "NFCI_ret_5d", "HUM_Humana_ret_5d", "heston_ev_h3", "EWL_Switzerland_vol_20d", "HangSeng_HK_ret_5d", "GILD_Gilead_ret_20d", "PPL_PPL_ret_1d", "PCAR_PaccarInc_ret_5d", "DOW_Price_zscore_60d", "US1Y_Rate_ret_20d", "gjr_condvar_h1", "PLD_Prologis_ret_5d", "US3M_Rate_zscore_60d", "IWM_SmallCap_vol_20d", "XOM_ret_20d"], "is_new": true}, {"model_id": "new_h10_GLOBAL_LightGBM_N20_t1", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 10, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "MSTR_Bitcoin3_ret_20d", "EWC_Canada_zscore_60d", "ORCL_zscore_60d", "US6M_Rate_ret_20d", "DE_Deere_vol_20d", "EWM_Malaysia_ret_1d", "AMD_ret_5d", "DAX_Germany_vol_20d", "EWL_Switzerland_vol_20d", "PAYX_Paychex_zscore_60d", "ORCL_vol_20d", "TXN_vol_20d", "LUV_SouthwestAir_ret_5d", "US1Y_Rate_ret_5d", "DAX_Germany_zscore_60d", "DOW_Price_zscore_60d", "ENB_EnbridgeInc_ret_1d", "LOW_Lowes_ret_20d"], "is_new": true}, {"model_id": "new_h10_GLOBAL_LightGBM_N20_t2", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 10, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "MS_MorganStanley_zscore_60d", "IBEX_Spain_ret_20d", "HUM_Humana_ret_5d", "NVDA_vol_20d", "AORD_AUS_zscore_60d", "heston_var_ev_h7", "spx_vol_5d", "Brent_Oil_FRED_ret_5d", "CCI_CrownCastle_vol_20d", "EWY_Korea_ret_20d", "XOM_ret_20d", "DE_Deere_vol_20d", "IWM_SmallCap_vol_20d", "AMT_AmericanTower_ret_1d", "EOG_EOGResources_vol_20d", "CMCSA_ret_1d", "EWL_Switzerland_vol_20d", "LLY_zscore_60d"], "is_new": true}, {"model_id": "new_h10_GLOBAL_LightGBM_N20_t3", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 10, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "XLK_Tech_zscore_60d", "XLF_Fin_vol_20d", "BA_ret_1d", "EOG_EOGResources_vol_20d", "PCAR_PaccarInc_ret_5d", "XOM_ret_20d", "EWM_Malaysia_vol_20d", "spx_vol_5d", "NEE_NextEra_ret_20d", "PAYX_Paychex_zscore_60d", "NOC_Northrop_ret_20d", "SJM_JM_Smucker_ret_1d", "AORD_AUS_zscore_60d", "TM_Telephone_ret_1d", "heston_var_ev_h3", "NFCI_ret_5d", "US3M_Rate_zscore_60d", "spx_abs_ret_max_5d"], "is_new": true}, {"model_id": "new_h10_GLOBAL_LightGBM_N20_t4", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 10, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "TM_Telephone_vol_20d", "BLK_BlackRock_zscore_60d", "DHR_ret_1d", "heston_var_ev_h5", "SO_SouthernCo_ret_5d", "DHR_vol_20d", "SPY_zscore_60d", "SBUX_ret_5d", "CMCSA_ret_1d", "AMGN_Amgen_ret_1d", "IWM_SmallCap_vol_20d", "EWM_Malaysia_zscore_60d", "3M_vol_20d", "EWG_Germany_vol_20d", "MS_MorganStanley_ret_1d", "IYR_US_REIT2_zscore_60d", "EWY_Korea_ret_20d", "XLY_Disc_vol_20d"], "is_new": true}, {"model_id": "new_h10_GLOBAL_LightGBM_N20_t5", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 10, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "CLX_Clorox_vol_20d", "NOC_Northrop_ret_20d", "VRP_ma5", "BTI_BritishAmerican_ret_5d", "EWL_Switzerland_vol_20d", "spx_abs_ret_max_5d", "SO_SouthernCo_ret_5d", "TED_Spread_zscore_60d", "spx_vol_5d", "LOW_Lowes_ret_20d", "EWC_Canada_zscore_60d", "ES_Evergy_ret_1d", "AMT_AmericanTower_ret_1d", "SLB_Schlumberger_ret_1d", "SPY_zscore_60d", "T10Y2Y_Spread_ret_5d", "GD_GeneralDynamics_zscore_60d", "LUV_SouthwestAir_ret_5d"], "is_new": true}, {"model_id": "new_h10_GLOBAL_LightGBM_N20_t6", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 10, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "MS_MorganStanley_ret_1d", "ORCL_zscore_60d", "3M_vol_20d", "LLY_zscore_60d", "3M_ret_5d", "AMD_ret_5d", "MSTR_Bitcoin3_ret_5d", "US1Y_Rate_ret_20d", "MS_MorganStanley_zscore_60d", "EOG_EOGResources_vol_20d", "PAYX_Paychex_vol_20d", "EWL_Switzerland_zscore_60d", "SJM_JM_Smucker_ret_1d", "IYM_BasicMaterials_ret_20d", "DE_Deere_vol_20d", "VRP_ma5", "AXP_Amex_vol_20d", "NOC_Northrop_ret_20d"], "is_new": true}, {"model_id": "new_h10_GLOBAL_LightGBM_N20_t7", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 10, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "SBUX_zscore_60d", "heston_var_ev_h7", "US3M_Rate_vol_20d", "heston_ev_h3", "Retail_Sales_zscore_60d", "IYM_BasicMaterials_ret_20d", "HangSeng_HK_ret_5d", "BDX_Becton_Dickinson_ret_20d", "LOW_Lowes_ret_20d", "EWA_Australia_zscore_60d", "CI_Cigna_vol_20d", "XLK_Tech_zscore_60d", "GD_GeneralDynamics_zscore_60d", "ASX_Australia_vol_20d", "XLV_Health_zscore_60d", "MSTR_Bitcoin3_ret_20d", "PG_ret_20d", "VRP_ma5"], "is_new": true}, {"model_id": "new_h10_GLOBAL_LightGBM_N25_t0", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 10, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "SLB_Schlumberger_ret_5d", "DHR_vol_20d", "HD_ret_1d", "heston_var_ev_h5", "EWQ_France_ret_20d", "HD_ret_5d", "EWA_Australia_ret_1d", "CPB_CampbellSoup_vol_20d", "TED_Spread_zscore_60d", "AMD_ret_1d", "EQIX_Equinix_ret_5d", "Brent_Oil_FRED_ret_20d", "EWA_Australia_zscore_60d", "AMT_AmericanTower_ret_1d", "US7Y_Rate_ret_20d", "DIS_vol_20d", "ITT_ITTInc_ret_5d", "Nikkei_Japan_zscore_60d", "ORCL_vol_20d", "MSTR_Bitcoin3_ret_5d", "EWQ_France_zscore_60d", "CPB_CampbellSoup_ret_20d", "SJM_JM_Smucker_ret_1d"], "is_new": true}, {"model_id": "new_h10_GLOBAL_LightGBM_N25_t1", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 10, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "US1Y_Rate_ret_5d", "Nikkei_Japan_vol_20d", "AXP_Amex_ret_20d", "SBUX_ret_5d", "EWG_Germany_vol_20d", "AVB_AvalonBay_zscore_60d", "EWA_Australia_zscore_60d", "Core_CPI_zscore_60d", "ORCL_vol_20d", "EWM_Malaysia_zscore_60d", "US3M_Rate_vol_20d", "LOW_Lowes_ret_5d", "VOD_Vodafone_zscore_60d", "heston_var_ev_h5", "CPB_CampbellSoup_zscore_60d", "SJM_JM_Smucker_ret_5d", "gjr_condvar_h1", "MSTR_Bitcoin3_ret_5d", "CLX_Clorox_vol_20d", "TED_Spread_zscore_60d", "HangSeng_HK_ret_5d", "TXN_vol_20d", "heston_var_ev_h7"], "is_new": true}, {"model_id": "new_h10_GLOBAL_LightGBM_N25_t2", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 10, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "AXP_Amex_ret_20d", "SBUX_zscore_60d", "TM_Telephone_vol_20d", "EWA_Australia_ret_1d", "vix_mean_abs_ret_5d", "AXP_Amex_vol_20d", "ORCL_vol_20d", "M_Macys_vol_20d", "LOW_Lowes_ret_20d", "EWA_Australia_zscore_60d", "EWL_Switzerland_zscore_60d", "MO_AltriaMG_ret_1d", "MS_MorganStanley_ret_1d", "QQQ_vol_20d", "DIS_vol_20d", "EFFR_vol_20d", "HD_zscore_60d", "EWL_Switzerland_vol_20d", "TGT_Target_zscore_60d", "EWG_Germany_vol_20d", "SLB_Schlumberger_ret_5d", "US1Y_Rate_ret_5d", "US6M_Rate_ret_20d"], "is_new": true}, {"model_id": "new_h10_GLOBAL_LightGBM_N25_t3", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 10, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "DAX_Germany_vol_20d", "CPB_CampbellSoup_ret_5d", "Michigan_Sentiment_ret_20d", "AMD_ret_5d", "XLY_Disc_vol_20d", "CPB_CampbellSoup_zscore_60d", "HUM_Humana_ret_5d", "LLY_zscore_60d", "EWH_HongKong_ret_5d", "spx_momentum_3d", "spx_abs_ret_max_5d", "US6M_Rate_ret_20d", "M_Macys_vol_20d", "LMT_LockheedMartin_ret_1d", "FedFunds_zscore_60d", "EWA_Australia_zscore_60d", "CPB_CampbellSoup_ret_20d", "PAYX_Paychex_zscore_60d", "INTC_ret_5d", "EWM_Malaysia_zscore_60d", "AVB_AvalonBay_zscore_60d", "Industrial_Production_zscore_60d", "SCHW_Schwab_ret_5d"], "is_new": true}, {"model_id": "new_h10_GLOBAL_LightGBM_N25_t4", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 10, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "spx_abs_ret_max_5d", "AXP_Amex_vol_20d", "AORD_AUS_zscore_60d", "PFE_ret_1d", "EWG_Germany_ret_20d", "LOW_Lowes_ret_20d", "XLY_Disc_vol_20d", "AMZN_ret_5d", "heston_var_ev_h5", "PPL_PPL_ret_1d", "HangSeng_HK_ret_5d", "BA_ret_1d", "Core_CPI_zscore_60d", "EFFR_vol_20d", "EWM_Malaysia_vol_20d", "NWL_Newell_ret_20d", "MSTR_Bitcoin3_ret_5d", "MSTR_Bitcoin3_ret_20d", "gjr_condvar_h1", "US6M_Rate_ret_20d", "EFFR_ret_1d", "PCAR_PaccarInc_ret_5d", "JNJ_ret_1d"], "is_new": true}, {"model_id": "new_h10_GLOBAL_LightGBM_N25_t5", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 10, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWS_Singapore_ret_5d", "MSTR_Bitcoin3_ret_5d", "NVDA_vol_20d", "CCI_CrownCastle_vol_20d", "SJM_JM_Smucker_ret_1d", "Brent_Oil_FRED_ret_5d", "SLB_Schlumberger_ret_1d", "PAYX_Paychex_vol_20d", "SO_SouthernCo_ret_5d", "US3Y_Rate_ret_5d", "FedFunds_zscore_60d", "vix_acceleration_1d", "LLY_zscore_60d", "EOG_EOGResources_ret_5d", "US5Y_Rate_ret_5d", "VRP_ma5", "AMT_AmericanTower_ret_1d", "HangSeng_HK_ret_1d", "EWJ_Japan_vol_20d", "NWL_Newell_ret_20d", "PAYX_Paychex_ret_20d", "US1Y_Rate_ret_5d", "DE_Deere_vol_20d"], "is_new": true}, {"model_id": "new_h10_GLOBAL_LightGBM_N25_t6", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 10, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "ITT_ITTInc_ret_5d", "Michigan_Sentiment_ret_20d", "heston_var_ev_h7", "US3Y_Rate_ret_5d", "TED_Spread_zscore_60d", "PG_ret_20d", "TED_Spread_vol_20d", "NVDA_vol_20d", "EWG_Germany_ret_20d", "M_Macys_vol_20d", "ORCL_zscore_60d", "AXP_Amex_ret_20d", "PCAR_PaccarInc_ret_5d", "DOW_Price_zscore_60d", "IBEX_Spain_ret_20d", "Core_PCE_zscore_60d", "EWA_Australia_ret_1d", "XLF_Fin_vol_20d", "HangSeng_HK_ret_5d", "EXC_Exelon_ret_1d", "JNJ_ret_1d", "heston_var_ev_h3", "PPL_PPL_ret_1d"], "is_new": true}, {"model_id": "new_h10_GLOBAL_LightGBM_N25_t7", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 10, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "IYM_BasicMaterials_ret_20d", "INTC_ret_5d", "NWL_Newell_ret_20d", "DOW_Price_zscore_60d", "EOG_EOGResources_vol_20d", "XOM_ret_1d", "EQIX_Equinix_ret_5d", "EWH_HongKong_ret_5d", "QQQ_vol_20d", "LUV_SouthwestAir_ret_5d", "EOG_EOGResources_ret_5d", "PAYX_Paychex_ret_20d", "heston_var_ev_h3", "XLV_Health_zscore_60d", "XOM_ret_20d", "INTC_ret_1d", "TED_Spread_vol_20d", "SPY_zscore_60d", "PPL_PPL_ret_1d", "3M_vol_20d", "Nikkei_Japan_zscore_60d", "HD_ret_20d", "IBEX_Spain_ret_20d"], "is_new": true}, {"model_id": "new_h10_GLOBAL_LightGBM_N30_t0", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 10, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "BA_ret_1d", "spx_momentum_3d", "DE_Deere_vol_20d", "EWL_Switzerland_zscore_60d", "SBUX_zscore_60d", "ITT_ITTInc_ret_5d", "heston_var_ev_h3", "MO_AltriaMG_ret_1d", "TED_Spread_vol_20d", "PAYX_Paychex_vol_20d", "MS_MorganStanley_ret_5d", "PFE_ret_1d", "MSTR_Bitcoin3_ret_5d", "AMD_ret_5d", "Core_CPI_zscore_60d", "heston_ev_h3", "EOG_EOGResources_ret_5d", "Brent_Oil_FRED_ret_20d", "TXN_vol_20d", "EFFR_ret_1d", "XLB_Materials_zscore_60d", "US3M_Rate_zscore_60d", "SJM_JM_Smucker_ret_5d", "EWG_Germany_vol_20d", "ORCL_vol_20d", "DAX_Germany_zscore_60d", "CLX_Clorox_vol_20d", "US5Y_Rate_ret_5d"], "is_new": true}, {"model_id": "new_h10_GLOBAL_LightGBM_N30_t1", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 10, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "vix_mean_abs_ret_5d", "EWL_Switzerland_zscore_60d", "PAYX_Paychex_vol_20d", "LOW_Lowes_ret_20d", "spx_vol_5d", "MS_MorganStanley_zscore_60d", "XOM_ret_20d", "CCI_CrownCastle_vol_20d", "CPB_CampbellSoup_zscore_60d", "TED_Spread_vol_20d", "EOG_EOGResources_vol_20d", "heston_var_ev_h7", "heston_var_ev_h5", "3M_vol_20d", "IYR_US_REIT2_zscore_60d", "SJM_JM_Smucker_ret_1d", "SLB_Schlumberger_ret_1d", "BTI_BritishAmerican_ret_5d", "INTC_ret_1d", "vix_acceleration_1d", "SLB_Schlumberger_ret_5d", "EOG_EOGResources_ret_5d", "MRK_Merck_zscore_60d", "WTI_Oil_FRED_zscore_60d", "EWM_Malaysia_ret_1d", "EXC_Exelon_ret_1d", "NEE_NextEra_ret_20d", "XOM_ret_1d"], "is_new": true}, {"model_id": "new_h10_GLOBAL_LightGBM_N30_t2", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 10, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "BA_ret_1d", "Industrial_Production_zscore_60d", "EWY_Korea_zscore_60d", "AMD_ret_5d", "EOG_EOGResources_vol_20d", "vix_acceleration_1d", "DHR_ret_1d", "AMGN_Amgen_ret_1d", "HangSeng_HK_vol_20d", "LOW_Lowes_ret_20d", "PG_ret_20d", "US6M_Rate_ret_20d", "HD_ret_1d", "PCAR_PaccarInc_ret_5d", "HD_ret_5d", "SLB_Schlumberger_ret_1d", "CI_Cigna_vol_20d", "MSTR_Bitcoin3_ret_5d", "EWQ_France_zscore_60d", "MS_MorganStanley_zscore_60d", "CPB_CampbellSoup_ret_20d", "ORCL_zscore_60d", "DOW_Price_zscore_60d", "MSTR_Bitcoin3_ret_20d", "SJM_JM_Smucker_ret_1d", "Brent_Oil_FRED_ret_20d", "VOD_Vodafone_zscore_60d", "TXN_vol_20d"], "is_new": true}, {"model_id": "new_h10_GLOBAL_LightGBM_N30_t3", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 10, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "XLY_Disc_vol_20d", "HangSeng_HK_ret_5d", "HD_ret_5d", "IBEX_Spain_ret_20d", "PCAR_PaccarInc_ret_5d", "Retail_Sales_zscore_60d", "DHR_ret_1d", "EXC_Exelon_ret_1d", "EWQ_France_ret_20d", "EWG_Germany_vol_20d", "JNJ_ret_1d", "GE_ret_1d", "heston_var_ev_h5", "SPY_zscore_60d", "spx_vol_5d", "GD_GeneralDynamics_zscore_60d", "DE_Deere_ret_5d", "EWM_Malaysia_zscore_60d", "US7Y_Rate_ret_20d", "AMZN_ret_5d", "DE_Deere_vol_20d", "T_ret_1d", "SBUX_vol_20d", "US3M_Rate_zscore_60d", "SLB_Schlumberger_ret_1d", "LMT_LockheedMartin_vol_20d", "EFFR_ret_1d", "NOC_Northrop_ret_20d"], "is_new": true}, {"model_id": "new_h10_GLOBAL_LightGBM_N30_t4", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 10, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "US3M_Rate_zscore_60d", "T_ret_1d", "EFFR_ret_1d", "XLK_Tech_zscore_60d", "HD_zscore_60d", "DE_Deere_vol_20d", "US3Y_Rate_ret_5d", "SLB_Schlumberger_ret_5d", "XOM_ret_1d", "SCHW_Schwab_ret_5d", "Brent_Oil_FRED_ret_20d", "EWY_Korea_ret_20d", "HUM_Humana_ret_5d", "ASX_Australia_ret_5d", "EWL_Switzerland_zscore_60d", "EWA_Australia_ret_1d", "LMT_LockheedMartin_ret_1d", "SJM_JM_Smucker_ret_1d", "TXN_vol_20d", "DHR_vol_20d", "SO_SouthernCo_ret_5d", "EXC_Exelon_ret_1d", "EQR_Equity_ret_1d", "SBUX_ret_5d", "LUV_SouthwestAir_ret_5d", "MO_AltriaMG_ret_1d", "PG_ret_20d", "CPB_CampbellSoup_zscore_60d"], "is_new": true}, {"model_id": "new_h10_GLOBAL_LightGBM_N30_t5", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 10, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "ASX_Australia_vol_20d", "EFFR_vol_20d", "XLF_Fin_vol_20d", "spx_vol_5d", "DIS_vol_20d", "IYR_US_REIT2_zscore_60d", "US3M_Rate_vol_20d", "TXN_vol_20d", "EWQ_France_zscore_60d", "EXC_Exelon_ret_1d", "Nikkei_Japan_zscore_60d", "US7Y_Rate_ret_20d", "MS_MorganStanley_ret_1d", "HUM_Humana_ret_5d", "US30Y_Rate_ret_20d", "TM_Telephone_vol_20d", "MRK_Merck_zscore_60d", "CLX_Clorox_vol_20d", "EWA_Australia_ret_1d", "heston_var_ev_h7", "EWM_Malaysia_ret_1d", "AXP_Amex_ret_20d", "ES_Evergy_ret_1d", "PFE_ret_1d", "EOG_EOGResources_vol_20d", "US1Y_Rate_ret_20d", "MSTR_Bitcoin3_ret_20d", "AMD_ret_1d"], "is_new": true}, {"model_id": "new_h10_GLOBAL_LightGBM_N30_t6", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 10, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "AXP_Amex_vol_20d", "ITT_ITTInc_ret_5d", "HangSeng_HK_ret_1d", "Nikkei_Japan_vol_20d", "EWA_Australia_zscore_60d", "WTI_Oil_FRED_zscore_60d", "US3M_Rate_vol_20d", "XLB_Materials_zscore_60d", "TXN_vol_20d", "heston_var_ev_h5", "DAX_Germany_vol_20d", "NWL_Newell_ret_20d", "CMCSA_ret_1d", "VRP_ma5", "vix_mean_abs_ret_5d", "LOW_Lowes_ret_20d", "TED_Spread_vol_20d", "gjr_condvar_h1", "heston_ev_h3", "spx_abs_ret_max_5d", "EWQ_France_zscore_60d", "SBUX_vol_20d", "US7Y_Rate_ret_20d", "VOD_Vodafone_zscore_60d", "GD_GeneralDynamics_zscore_60d", "EWM_Malaysia_ret_1d", "AMT_AmericanTower_ret_1d", "XOM_ret_1d"], "is_new": true}, {"model_id": "new_h10_GLOBAL_LightGBM_N30_t7", "algo": "LightGBM", "regime": "GLOBAL", "horizon": 10, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "SLB_Schlumberger_ret_5d", "XLF_Fin_vol_20d", "BTI_BritishAmerican_ret_20d", "DHR_vol_20d", "BLK_BlackRock_zscore_60d", "AMD_ret_1d", "US7Y_Rate_ret_20d", "AMGN_Amgen_ret_1d", "TM_Telephone_ret_1d", "heston_ev_h3", "TXN_vol_20d", "EWM_Malaysia_ret_1d", "QQQ_vol_20d", "LLY_zscore_60d", "US3M_Rate_zscore_60d", "PG_ret_20d", "AORD_AUS_zscore_60d", "EOG_EOGResources_vol_20d", "M_Macys_vol_20d", "EWC_Canada_zscore_60d", "XLY_Disc_vol_20d", "EWJ_Japan_vol_20d", "Retail_Sales_zscore_60d", "Brent_Oil_FRED_ret_20d", "PFE_ret_1d", "PLD_Prologis_ret_5d", "INTC_ret_1d", "BA_ret_1d"], "is_new": true}, {"model_id": "new_h10_GLOBAL_GradientBoosting_N5_t0", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 10, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "spx_momentum_3d", "SCHW_Schwab_ret_5d", "PPL_PPL_ret_1d"], "is_new": true}, {"model_id": "new_h10_GLOBAL_GradientBoosting_N5_t1", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 10, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "AXP_Amex_vol_20d", "vix_mean_abs_ret_5d", "ORCL_zscore_60d"], "is_new": true}, {"model_id": "new_h10_GLOBAL_GradientBoosting_N5_t2", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 10, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "Core_CPI_zscore_60d", "heston_ev_h3", "EXC_Exelon_ret_1d"], "is_new": true}, {"model_id": "new_h10_GLOBAL_GradientBoosting_N5_t3", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 10, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "DOW_Price_zscore_60d", "EWY_Korea_zscore_60d", "heston_ev_h3"], "is_new": true}, {"model_id": "new_h10_GLOBAL_GradientBoosting_N5_t4", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 10, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "NFCI_ret_5d", "US3M_Rate_vol_20d", "WTI_Oil_FRED_zscore_60d"], "is_new": true}, {"model_id": "new_h10_GLOBAL_GradientBoosting_N5_t5", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 10, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "LLY_zscore_60d", "EWM_Malaysia_vol_20d", "EWL_Switzerland_zscore_60d"], "is_new": true}, {"model_id": "new_h10_GLOBAL_GradientBoosting_N5_t6", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 10, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "IBEX_Spain_ret_20d", "CTAS_Cintas_vol_20d", "EWS_Singapore_ret_5d"], "is_new": true}, {"model_id": "new_h10_GLOBAL_GradientBoosting_N5_t7", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 10, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "hmm_p_stress", "Retail_Sales_zscore_60d", "GD_GeneralDynamics_zscore_60d"], "is_new": true}, {"model_id": "new_h10_GLOBAL_GradientBoosting_N8_t0", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 10, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "T10Y2Y_Spread_ret_5d", "PFE_ret_1d", "PPL_PPL_ret_1d", "EWS_Singapore_ret_5d", "TM_Telephone_vol_20d", "EOG_EOGResources_vol_20d"], "is_new": true}, {"model_id": "new_h10_GLOBAL_GradientBoosting_N8_t1", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 10, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWG_Germany_vol_20d", "EFFR_vol_20d", "AORD_AUS_zscore_60d", "XLV_Health_zscore_60d", "BA_ret_1d", "AMD_ret_1d"], "is_new": true}, {"model_id": "new_h10_GLOBAL_GradientBoosting_N8_t2", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 10, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "HUM_Humana_ret_5d", "Retail_Sales_zscore_60d", "DAX_Germany_vol_20d", "vix_acceleration_1d", "TM_Telephone_ret_1d", "XOM_ret_20d"], "is_new": true}, {"model_id": "new_h10_GLOBAL_GradientBoosting_N8_t3", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 10, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWA_Australia_ret_1d", "HangSeng_HK_vol_20d", "VRP_ma5", "BA_ret_1d", "DE_Deere_vol_20d", "INTC_ret_5d"], "is_new": true}, {"model_id": "new_h10_GLOBAL_GradientBoosting_N8_t4", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 10, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWG_Germany_ret_20d", "PLD_Prologis_ret_5d", "BA_ret_1d", "MSTR_Bitcoin3_ret_5d", "TED_Spread_vol_20d", "LOW_Lowes_ret_5d"], "is_new": true}, {"model_id": "new_h10_GLOBAL_GradientBoosting_N8_t5", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 10, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "HangSeng_HK_vol_20d", "EXC_Exelon_zscore_60d", "LOW_Lowes_ret_20d", "LOW_Lowes_ret_5d", "EWG_Germany_vol_20d", "PAYX_Paychex_zscore_60d"], "is_new": true}, {"model_id": "new_h10_GLOBAL_GradientBoosting_N8_t6", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 10, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "gjr_condvar_h1", "DE_Deere_ret_5d", "DHR_ret_1d", "SPY_zscore_60d", "XLK_Tech_zscore_60d", "EXC_Exelon_zscore_60d"], "is_new": true}, {"model_id": "new_h10_GLOBAL_GradientBoosting_N8_t7", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 10, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "MRK_Merck_zscore_60d", "gjr_condvar_h1", "EWS_Singapore_ret_5d", "IYM_BasicMaterials_ret_20d", "IBEX_Spain_ret_20d", "SJM_JM_Smucker_ret_1d"], "is_new": true}, {"model_id": "new_h10_GLOBAL_GradientBoosting_N10_t0", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 10, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "spx_vol_5d", "US1Y_Rate_ret_20d", "DHR_ret_1d", "CMCSA_ret_1d", "spx_abs_ret_max_5d", "SBUX_zscore_60d", "QQQ_vol_20d", "NVDA_vol_20d"], "is_new": true}, {"model_id": "new_h10_GLOBAL_GradientBoosting_N10_t1", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 10, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "LOW_Lowes_ret_5d", "SLB_Schlumberger_ret_1d", "BTI_BritishAmerican_ret_5d", "spx_vol_5d", "Brent_Oil_FRED_ret_5d", "hmm_p_stress", "CPB_CampbellSoup_ret_5d", "ES_Evergy_ret_1d"], "is_new": true}, {"model_id": "new_h10_GLOBAL_GradientBoosting_N10_t2", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 10, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "Retail_Sales_zscore_60d", "BLK_BlackRock_zscore_60d", "VOD_Vodafone_zscore_60d", "LUV_SouthwestAir_ret_5d", "PCAR_PaccarInc_ret_5d", "CPB_CampbellSoup_vol_20d", "HD_ret_5d", "MSTR_Bitcoin3_ret_20d"], "is_new": true}, {"model_id": "new_h10_GLOBAL_GradientBoosting_N10_t3", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 10, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "US1Y_Rate_ret_20d", "PAYX_Paychex_vol_20d", "vix_mean_abs_ret_5d", "US3M_Rate_zscore_60d", "MS_MorganStanley_zscore_60d", "CPB_CampbellSoup_vol_20d", "XLY_Disc_vol_20d", "DE_Deere_ret_5d"], "is_new": true}, {"model_id": "new_h10_GLOBAL_GradientBoosting_N10_t4", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 10, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "AXP_Amex_ret_20d", "ENB_EnbridgeInc_ret_1d", "gjr_condvar_h1", "NEE_NextEra_ret_20d", "HUM_Humana_ret_5d", "EWG_Germany_vol_20d", "XLF_Fin_vol_20d", "Core_PCE_zscore_60d"], "is_new": true}, {"model_id": "new_h10_GLOBAL_GradientBoosting_N10_t5", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 10, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "SLB_Schlumberger_ret_5d", "SBUX_ret_5d", "BDX_Becton_Dickinson_ret_20d", "PCAR_PaccarInc_ret_5d", "US7Y_Rate_ret_20d", "SBUX_zscore_60d", "SJM_JM_Smucker_ret_1d", "HangSeng_HK_vol_20d"], "is_new": true}, {"model_id": "new_h10_GLOBAL_GradientBoosting_N10_t6", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 10, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "AMD_ret_1d", "CPB_CampbellSoup_zscore_60d", "PCAR_PaccarInc_ret_5d", "AXP_Amex_ret_20d", "PAYX_Paychex_ret_20d", "heston_var_ev_h7", "heston_var_ev_h5", "IYM_BasicMaterials_ret_20d"], "is_new": true}, {"model_id": "new_h10_GLOBAL_GradientBoosting_N10_t7", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 10, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "spx_abs_ret_max_5d", "MSTR_Bitcoin3_ret_5d", "SPY_zscore_60d", "DOW_Price_zscore_60d", "PAYX_Paychex_vol_20d", "PAYX_Paychex_ret_20d", "Michigan_Sentiment_ret_20d", "LOW_Lowes_ret_5d"], "is_new": true}, {"model_id": "new_h10_GLOBAL_GradientBoosting_N12_t0", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 10, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "ASX_Australia_ret_5d", "PCAR_PaccarInc_ret_5d", "WTI_Oil_FRED_zscore_60d", "EWM_Malaysia_vol_20d", "MSTR_Bitcoin3_ret_1d", "LMT_LockheedMartin_ret_1d", "CPB_CampbellSoup_ret_5d", "ES_Evergy_ret_1d", "SBUX_ret_5d", "US3Y_Rate_ret_5d"], "is_new": true}, {"model_id": "new_h10_GLOBAL_GradientBoosting_N12_t1", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 10, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "HD_zscore_60d", "Nikkei_Japan_zscore_60d", "vix_mean_abs_ret_5d", "DIS_vol_20d", "LUV_SouthwestAir_ret_5d", "LMT_LockheedMartin_vol_20d", "spx_momentum_3d", "EWH_HongKong_ret_5d", "LLY_zscore_60d", "ORCL_vol_20d"], "is_new": true}, {"model_id": "new_h10_GLOBAL_GradientBoosting_N12_t2", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 10, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWL_Switzerland_vol_20d", "AMT_AmericanTower_ret_1d", "LOW_Lowes_ret_5d", "EOG_EOGResources_vol_20d", "AMD_ret_5d", "XOM_ret_1d", "MS_MorganStanley_ret_5d", "heston_var_ev_h5", "PPL_PPL_ret_1d", "EWS_Singapore_ret_5d"], "is_new": true}, {"model_id": "new_h10_GLOBAL_GradientBoosting_N12_t3", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 10, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "T10Y2Y_Spread_ret_5d", "MSTR_Bitcoin3_ret_5d", "DHR_vol_20d", "CI_Cigna_vol_20d", "CPB_CampbellSoup_zscore_60d", "Brent_Oil_FRED_ret_20d", "Industrial_Production_zscore_60d", "MO_AltriaMG_ret_1d", "NOC_Northrop_ret_20d", "XLB_Materials_zscore_60d"], "is_new": true}, {"model_id": "new_h10_GLOBAL_GradientBoosting_N12_t4", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 10, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EXC_Exelon_zscore_60d", "FedFunds_zscore_60d", "HD_ret_1d", "PCAR_PaccarInc_ret_5d", "AMGN_Amgen_ret_1d", "EWQ_France_zscore_60d", "EWG_Germany_ret_20d", "SCHW_Schwab_ret_5d", "EWA_Australia_zscore_60d", "AMD_ret_5d"], "is_new": true}, {"model_id": "new_h10_GLOBAL_GradientBoosting_N12_t5", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 10, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "ITT_ITTInc_ret_5d", "DOW_Price_zscore_60d", "EWQ_France_zscore_60d", "HangSeng_HK_ret_5d", "US1Y_Rate_ret_5d", "BTI_BritishAmerican_ret_5d", "EWQ_France_ret_20d", "IYR_US_REIT2_zscore_60d", "AMZN_ret_5d", "spx_momentum_3d"], "is_new": true}, {"model_id": "new_h10_GLOBAL_GradientBoosting_N12_t6", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 10, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "hmm_p_stress", "EWA_Australia_zscore_60d", "DIS_vol_20d", "US1Y_Rate_ret_20d", "DAX_Germany_zscore_60d", "AXP_Amex_vol_20d", "PFE_ret_1d", "AMD_ret_5d", "MS_MorganStanley_zscore_60d", "PLD_Prologis_ret_5d"], "is_new": true}, {"model_id": "new_h10_GLOBAL_GradientBoosting_N12_t7", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 10, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "PFE_ret_1d", "T_ret_1d", "CPB_CampbellSoup_ret_5d", "MSTR_Bitcoin3_ret_5d", "Core_PCE_zscore_60d", "CCI_CrownCastle_vol_20d", "vix_mean_abs_ret_5d", "AORD_AUS_zscore_60d", "XLF_Fin_vol_20d", "hmm_p_stress"], "is_new": true}, {"model_id": "new_h10_GLOBAL_GradientBoosting_N15_t0", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 10, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "DHR_vol_20d", "US5Y_Rate_ret_5d", "INTC_ret_1d", "EFFR_vol_20d", "CTAS_Cintas_vol_20d", "DAX_Germany_zscore_60d", "EQIX_Equinix_ret_5d", "heston_var_ev_h5", "CPB_CampbellSoup_ret_20d", "LMT_LockheedMartin_vol_20d", "XLF_Fin_vol_20d", "ES_Evergy_ret_1d", "CCI_CrownCastle_vol_20d"], "is_new": true}, {"model_id": "new_h10_GLOBAL_GradientBoosting_N15_t1", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 10, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "TED_Spread_zscore_60d", "EQR_Equity_ret_1d", "NOC_Northrop_ret_20d", "NFCI_ret_5d", "EFFR_vol_20d", "WTI_Oil_FRED_zscore_60d", "VVIX_ret_20d", "GILD_Gilead_ret_20d", "PAYX_Paychex_zscore_60d", "AMGN_Amgen_ret_1d", "EWL_Switzerland_zscore_60d", "BTI_BritishAmerican_ret_20d", "EWM_Malaysia_zscore_60d"], "is_new": true}, {"model_id": "new_h10_GLOBAL_GradientBoosting_N15_t2", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 10, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "GE_ret_1d", "AMGN_Amgen_ret_1d", "VOD_Vodafone_zscore_60d", "SLB_Schlumberger_ret_5d", "IYM_BasicMaterials_ret_20d", "Core_CPI_zscore_60d", "EWY_Korea_zscore_60d", "EWQ_France_zscore_60d", "CMCSA_ret_1d", "MS_MorganStanley_ret_1d", "FedFunds_zscore_60d", "US5Y_Rate_ret_5d", "BTI_BritishAmerican_ret_5d"], "is_new": true}, {"model_id": "new_h10_GLOBAL_GradientBoosting_N15_t3", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 10, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "HangSeng_HK_ret_5d", "WTI_Oil_FRED_zscore_60d", "MSTR_Bitcoin3_ret_20d", "EWC_Canada_zscore_60d", "EXC_Exelon_ret_1d", "Core_CPI_zscore_60d", "SJM_JM_Smucker_ret_5d", "Michigan_Sentiment_ret_20d", "VOD_Vodafone_zscore_60d", "US1Y_Rate_ret_5d", "NVDA_vol_20d", "NEE_NextEra_ret_20d", "SJM_JM_Smucker_ret_1d"], "is_new": true}, {"model_id": "new_h10_GLOBAL_GradientBoosting_N15_t4", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 10, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "MO_AltriaMG_ret_1d", "EQIX_Equinix_ret_5d", "US30Y_Rate_ret_20d", "BA_ret_1d", "INTC_ret_1d", "PAYX_Paychex_ret_20d", "EXC_Exelon_zscore_60d", "CPB_CampbellSoup_ret_20d", "EFFR_vol_20d", "IBEX_Spain_ret_20d", "PAYX_Paychex_vol_20d", "M_Macys_vol_20d", "SPY_zscore_60d"], "is_new": true}, {"model_id": "new_h10_GLOBAL_GradientBoosting_N15_t5", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 10, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "MO_AltriaMG_ret_1d", "US3Y_Rate_ret_5d", "ASX_Australia_ret_5d", "heston_var_ev_h5", "SBUX_vol_20d", "MSTR_Bitcoin3_ret_5d", "DHR_ret_1d", "XOM_ret_1d", "gjr_condvar_h1", "EXC_Exelon_ret_1d", "MSTR_Bitcoin3_ret_1d", "EWY_Korea_ret_20d", "Nikkei_Japan_zscore_60d"], "is_new": true}, {"model_id": "new_h10_GLOBAL_GradientBoosting_N15_t6", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 10, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "ENB_EnbridgeInc_ret_1d", "DE_Deere_vol_20d", "WTI_Oil_FRED_zscore_60d", "EOG_EOGResources_vol_20d", "NFCI_ret_5d", "FedFunds_zscore_60d", "XLV_Health_zscore_60d", "T10Y2Y_Spread_ret_5d", "MSTR_Bitcoin3_ret_5d", "SPY_zscore_60d", "EWJ_Japan_vol_20d", "3M_vol_20d", "heston_var_ev_h5"], "is_new": true}, {"model_id": "new_h10_GLOBAL_GradientBoosting_N15_t7", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 10, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "PAYX_Paychex_ret_20d", "EFFR_ret_1d", "heston_var_ev_h7", "ASX_Australia_vol_20d", "US6M_Rate_ret_20d", "AXP_Amex_vol_20d", "Brent_Oil_FRED_ret_20d", "CI_Cigna_vol_20d", "VVIX_ret_20d", "CTAS_Cintas_vol_20d", "EWC_Canada_zscore_60d", "PFE_ret_1d", "HD_zscore_60d"], "is_new": true}, {"model_id": "new_h10_GLOBAL_GradientBoosting_N20_t0", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 10, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "3M_ret_5d", "ES_Evergy_ret_1d", "hmm_p_stress", "CPB_CampbellSoup_ret_5d", "3M_vol_20d", "TED_Spread_vol_20d", "PAYX_Paychex_zscore_60d", "SPY_zscore_60d", "ORCL_zscore_60d", "HD_ret_20d", "FedFunds_zscore_60d", "TM_Telephone_ret_1d", "gjr_condvar_h1", "MRK_Merck_zscore_60d", "XLY_Disc_vol_20d", "NWL_Newell_ret_20d", "vix_acceleration_1d", "MS_MorganStanley_zscore_60d"], "is_new": true}, {"model_id": "new_h10_GLOBAL_GradientBoosting_N20_t1", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 10, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "GE_ret_1d", "Nikkei_Japan_zscore_60d", "XLF_Fin_vol_20d", "XOM_ret_1d", "Core_PCE_zscore_60d", "hmm_p_stress", "US5Y_Rate_ret_5d", "EWA_Australia_zscore_60d", "AORD_AUS_zscore_60d", "EWJ_Japan_vol_20d", "CMCSA_ret_1d", "MO_AltriaMG_ret_1d", "T10Y2Y_Spread_ret_5d", "IBEX_Spain_ret_20d", "DE_Deere_ret_5d", "JNJ_ret_1d", "NFCI_ret_5d", "BTI_BritishAmerican_ret_20d"], "is_new": true}, {"model_id": "new_h10_GLOBAL_GradientBoosting_N20_t2", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 10, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "CTAS_Cintas_vol_20d", "Brent_Oil_FRED_ret_20d", "INTC_ret_1d", "VVIX_ret_20d", "EQIX_Equinix_ret_5d", "HD_ret_1d", "QQQ_vol_20d", "ASX_Australia_vol_20d", "CMCSA_ret_1d", "IWM_SmallCap_vol_20d", "VRP_ma5", "TED_Spread_vol_20d", "AXP_Amex_vol_20d", "hmm_p_stress", "PAYX_Paychex_zscore_60d", "ES_Evergy_ret_1d", "JNJ_ret_1d", "heston_var_ev_h7"], "is_new": true}, {"model_id": "new_h10_GLOBAL_GradientBoosting_N20_t3", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 10, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "Brent_Oil_FRED_ret_20d", "AMZN_ret_5d", "XLB_Materials_zscore_60d", "SBUX_ret_5d", "SCHW_Schwab_ret_5d", "DHR_ret_1d", "VVIX_ret_20d", "QQQ_vol_20d", "CI_Cigna_vol_20d", "EWM_Malaysia_zscore_60d", "AMD_ret_1d", "MS_MorganStanley_zscore_60d", "spx_abs_ret_max_5d", "MS_MorganStanley_ret_5d", "NVDA_vol_20d", "PLD_Prologis_ret_5d", "US3M_Rate_zscore_60d", "US1Y_Rate_ret_5d"], "is_new": true}, {"model_id": "new_h10_GLOBAL_GradientBoosting_N20_t4", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 10, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "PPL_PPL_ret_1d", "LLY_zscore_60d", "XLF_Fin_vol_20d", "WTI_Oil_FRED_zscore_60d", "TXN_vol_20d", "TM_Telephone_vol_20d", "EFFR_ret_1d", "EFFR_vol_20d", "XOM_ret_1d", "XOM_ret_20d", "CCI_CrownCastle_vol_20d", "Nikkei_Japan_zscore_60d", "spx_momentum_3d", "AXP_Amex_vol_20d", "TED_Spread_vol_20d", "PG_ret_20d", "PAYX_Paychex_zscore_60d", "MS_MorganStanley_ret_1d"], "is_new": true}, {"model_id": "new_h10_GLOBAL_GradientBoosting_N20_t5", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 10, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "SLB_Schlumberger_ret_1d", "MS_MorganStanley_zscore_60d", "NFCI_ret_5d", "DIS_vol_20d", "XLF_Fin_vol_20d", "XLB_Materials_zscore_60d", "T10Y2Y_Spread_ret_5d", "BTI_BritishAmerican_ret_20d", "SO_SouthernCo_ret_5d", "CPB_CampbellSoup_ret_20d", "IYM_BasicMaterials_ret_20d", "T_ret_1d", "EQIX_Equinix_ret_5d", "MSTR_Bitcoin3_ret_20d", "DHR_vol_20d", "HD_zscore_60d", "EWL_Switzerland_vol_20d", "WTI_Oil_FRED_zscore_60d"], "is_new": true}, {"model_id": "new_h10_GLOBAL_GradientBoosting_N20_t6", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 10, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "CI_Cigna_vol_20d", "AXP_Amex_vol_20d", "VRP_ma5", "ASX_Australia_ret_5d", "TED_Spread_vol_20d", "EWG_Germany_vol_20d", "vix_mean_abs_ret_5d", "XOM_ret_20d", "QQQ_vol_20d", "SBUX_zscore_60d", "PAYX_Paychex_zscore_60d", "INTC_ret_5d", "ITT_ITTInc_ret_5d", "PAYX_Paychex_ret_20d", "US5Y_Rate_ret_5d", "XLY_Disc_vol_20d", "hmm_p_stress", "EWM_Malaysia_vol_20d"], "is_new": true}, {"model_id": "new_h10_GLOBAL_GradientBoosting_N20_t7", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 10, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "AMT_AmericanTower_ret_1d", "spx_vol_5d", "VRP_ma5", "EWA_Australia_ret_1d", "EWL_Switzerland_zscore_60d", "LOW_Lowes_ret_20d", "EWM_Malaysia_vol_20d", "EOG_EOGResources_vol_20d", "US3Y_Rate_ret_5d", "heston_var_ev_h3", "MSTR_Bitcoin3_ret_5d", "CPB_CampbellSoup_vol_20d", "BLK_BlackRock_zscore_60d", "US3M_Rate_zscore_60d", "CCI_CrownCastle_vol_20d", "XOM_ret_20d", "PPL_PPL_ret_1d", "hmm_p_stress"], "is_new": true}, {"model_id": "new_h10_GLOBAL_GradientBoosting_N25_t0", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 10, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWG_Germany_ret_20d", "SLB_Schlumberger_ret_1d", "US6M_Rate_ret_20d", "US7Y_Rate_ret_20d", "EWM_Malaysia_zscore_60d", "3M_vol_20d", "SPY_zscore_60d", "EOG_EOGResources_vol_20d", "SCHW_Schwab_ret_5d", "NVDA_vol_20d", "GE_ret_1d", "BTI_BritishAmerican_ret_20d", "ES_Evergy_ret_1d", "SJM_JM_Smucker_ret_1d", "PPL_PPL_ret_1d", "SLB_Schlumberger_ret_5d", "BTI_BritishAmerican_ret_5d", "INTC_ret_5d", "PCAR_PaccarInc_ret_5d", "LMT_LockheedMartin_ret_1d", "VOD_Vodafone_zscore_60d", "EQIX_Equinix_ret_5d", "BLK_BlackRock_zscore_60d"], "is_new": true}, {"model_id": "new_h10_GLOBAL_GradientBoosting_N25_t1", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 10, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "ES_Evergy_ret_1d", "SCHW_Schwab_ret_5d", "XLK_Tech_zscore_60d", "AMD_ret_1d", "EQIX_Equinix_ret_5d", "EWC_Canada_zscore_60d", "DAX_Germany_zscore_60d", "US5Y_Rate_ret_5d", "MSTR_Bitcoin3_ret_1d", "CPB_CampbellSoup_zscore_60d", "DHR_vol_20d", "ITT_ITTInc_ret_5d", "CI_Cigna_vol_20d", "AXP_Amex_ret_20d", "EWY_Korea_ret_20d", "Brent_Oil_FRED_ret_20d", "NEE_NextEra_ret_20d", "PFE_ret_1d", "AMZN_ret_5d", "TM_Telephone_vol_20d", "XLF_Fin_vol_20d", "DAX_Germany_vol_20d", "T10Y2Y_Spread_ret_5d"], "is_new": true}, {"model_id": "new_h10_GLOBAL_GradientBoosting_N25_t2", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 10, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "PFE_ret_1d", "CPB_CampbellSoup_vol_20d", "US6M_Rate_ret_20d", "heston_ev_h3", "TM_Telephone_vol_20d", "INTC_ret_1d", "spx_abs_ret_max_5d", "TM_Telephone_ret_1d", "Nikkei_Japan_vol_20d", "NWL_Newell_ret_20d", "Brent_Oil_FRED_ret_5d", "MRK_Merck_zscore_60d", "EXC_Exelon_ret_1d", "LOW_Lowes_ret_5d", "LMT_LockheedMartin_vol_20d", "ASX_Australia_vol_20d", "hmm_p_stress", "IWM_SmallCap_vol_20d", "XLF_Fin_vol_20d", "AMZN_ret_5d", "CMCSA_ret_1d", "PCAR_PaccarInc_ret_5d", "Nikkei_Japan_zscore_60d"], "is_new": true}, {"model_id": "new_h10_GLOBAL_GradientBoosting_N25_t3", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 10, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "VOD_Vodafone_zscore_60d", "US6M_Rate_ret_20d", "PAYX_Paychex_vol_20d", "US7Y_Rate_ret_20d", "heston_ev_h3", "EWL_Switzerland_zscore_60d", "MSTR_Bitcoin3_ret_5d", "SLB_Schlumberger_ret_5d", "SBUX_zscore_60d", "ENB_EnbridgeInc_ret_1d", "heston_var_ev_h5", "PFE_ret_1d", "TED_Spread_zscore_60d", "SJM_JM_Smucker_ret_5d", "XOM_ret_20d", "GD_GeneralDynamics_zscore_60d", "MSTR_Bitcoin3_ret_20d", "HUM_Humana_ret_5d", "EWY_Korea_ret_20d", "AMT_AmericanTower_ret_1d", "SJM_JM_Smucker_ret_1d", "CCI_CrownCastle_vol_20d", "TM_Telephone_vol_20d"], "is_new": true}, {"model_id": "new_h10_GLOBAL_GradientBoosting_N25_t4", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 10, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "Nikkei_Japan_zscore_60d", "SBUX_ret_5d", "NEE_NextEra_ret_20d", "FedFunds_zscore_60d", "vix_acceleration_1d", "PFE_ret_1d", "BTI_BritishAmerican_ret_5d", "US3M_Rate_zscore_60d", "NFCI_ret_5d", "CMCSA_ret_1d", "INTC_ret_1d", "heston_var_ev_h7", "EWM_Malaysia_vol_20d", "EWL_Switzerland_vol_20d", "LUV_SouthwestAir_ret_5d", "XLK_Tech_zscore_60d", "GE_ret_1d", "MS_MorganStanley_ret_5d", "SBUX_zscore_60d", "MRK_Merck_zscore_60d", "Michigan_Sentiment_ret_20d", "EOG_EOGResources_ret_5d", "LOW_Lowes_ret_20d"], "is_new": true}, {"model_id": "new_h10_GLOBAL_GradientBoosting_N25_t5", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 10, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "ES_Evergy_ret_1d", "T10Y2Y_Spread_ret_5d", "SJM_JM_Smucker_ret_1d", "SO_SouthernCo_ret_5d", "SBUX_ret_5d", "GILD_Gilead_ret_20d", "PPL_PPL_ret_1d", "EWA_Australia_ret_1d", "DOW_Price_zscore_60d", "MSTR_Bitcoin3_ret_1d", "FedFunds_zscore_60d", "EWG_Germany_ret_20d", "LOW_Lowes_ret_20d", "EWM_Malaysia_zscore_60d", "HangSeng_HK_ret_5d", "SBUX_zscore_60d", "CI_Cigna_vol_20d", "EWQ_France_zscore_60d", "spx_abs_ret_max_5d", "CMCSA_ret_1d", "ITT_ITTInc_ret_5d", "DAX_Germany_vol_20d", "CPB_CampbellSoup_ret_20d"], "is_new": true}, {"model_id": "new_h10_GLOBAL_GradientBoosting_N25_t6", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 10, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "LMT_LockheedMartin_ret_1d", "EWM_Malaysia_vol_20d", "CPB_CampbellSoup_ret_20d", "EWY_Korea_zscore_60d", "SCHW_Schwab_ret_5d", "BLK_BlackRock_zscore_60d", "HD_ret_20d", "EQR_Equity_ret_1d", "HD_zscore_60d", "US3M_Rate_vol_20d", "XLV_Health_zscore_60d", "EWA_Australia_ret_1d", "CI_Cigna_vol_20d", "Retail_Sales_zscore_60d", "TM_Telephone_ret_1d", "hmm_p_stress", "3M_ret_5d", "CPB_CampbellSoup_zscore_60d", "EWL_Switzerland_zscore_60d", "EWH_HongKong_ret_5d", "SO_SouthernCo_ret_5d", "XOM_ret_1d", "LOW_Lowes_ret_5d"], "is_new": true}, {"model_id": "new_h10_GLOBAL_GradientBoosting_N25_t7", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 10, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "NVDA_vol_20d", "TXN_vol_20d", "IBEX_Spain_ret_20d", "HD_ret_5d", "TGT_Target_zscore_60d", "Brent_Oil_FRED_ret_20d", "Core_PCE_zscore_60d", "PAYX_Paychex_ret_20d", "XLK_Tech_zscore_60d", "BTI_BritishAmerican_ret_20d", "PAYX_Paychex_vol_20d", "DE_Deere_ret_5d", "ASX_Australia_ret_5d", "EQIX_Equinix_ret_5d", "VOD_Vodafone_zscore_60d", "DE_Deere_vol_20d", "SBUX_vol_20d", "LMT_LockheedMartin_vol_20d", "US6M_Rate_ret_20d", "NFCI_ret_5d", "AVB_AvalonBay_zscore_60d", "DAX_Germany_zscore_60d", "spx_momentum_3d"], "is_new": true}, {"model_id": "new_h10_GLOBAL_GradientBoosting_N30_t0", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 10, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "ES_Evergy_ret_1d", "AXP_Amex_ret_20d", "PAYX_Paychex_ret_20d", "SO_SouthernCo_ret_5d", "BTI_BritishAmerican_ret_20d", "DE_Deere_ret_5d", "NVDA_vol_20d", "HD_ret_1d", "EWM_Malaysia_zscore_60d", "EWG_Germany_ret_20d", "LUV_SouthwestAir_ret_5d", "SBUX_ret_5d", "BLK_BlackRock_zscore_60d", "IWM_SmallCap_vol_20d", "CPB_CampbellSoup_ret_5d", "US30Y_Rate_ret_20d", "CCI_CrownCastle_vol_20d", "ITT_ITTInc_ret_5d", "PG_ret_20d", "US3Y_Rate_ret_5d", "ENB_EnbridgeInc_ret_1d", "EWS_Singapore_ret_5d", "Core_PCE_zscore_60d", "US6M_Rate_ret_20d", "EWQ_France_zscore_60d", "QQQ_vol_20d", "EQR_Equity_ret_1d", "AMT_AmericanTower_ret_1d"], "is_new": true}, {"model_id": "new_h10_GLOBAL_GradientBoosting_N30_t1", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 10, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "ASX_Australia_vol_20d", "spx_vol_5d", "AORD_AUS_zscore_60d", "Nikkei_Japan_vol_20d", "XOM_ret_20d", "HangSeng_HK_ret_1d", "Brent_Oil_FRED_ret_5d", "T_ret_1d", "MO_AltriaMG_ret_1d", "US3Y_Rate_ret_5d", "AMD_ret_1d", "EWM_Malaysia_ret_1d", "EWJ_Japan_vol_20d", "EMR_Emerson_ret_20d", "AVB_AvalonBay_zscore_60d", "EQR_Equity_ret_1d", "EWG_Germany_ret_20d", "PPL_PPL_ret_1d", "TED_Spread_zscore_60d", "spx_momentum_3d", "EWS_Singapore_ret_5d", "3M_vol_20d", "CTAS_Cintas_vol_20d", "HD_zscore_60d", "EQIX_Equinix_ret_5d", "heston_var_ev_h7", "CPB_CampbellSoup_ret_20d", "SBUX_ret_5d"], "is_new": true}, {"model_id": "new_h10_GLOBAL_GradientBoosting_N30_t2", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 10, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "XLY_Disc_vol_20d", "IYM_BasicMaterials_ret_20d", "GE_ret_1d", "AMGN_Amgen_ret_1d", "XLB_Materials_zscore_60d", "SLB_Schlumberger_ret_5d", "3M_vol_20d", "SBUX_zscore_60d", "CPB_CampbellSoup_vol_20d", "JNJ_ret_1d", "EOG_EOGResources_ret_5d", "GILD_Gilead_ret_20d", "BTI_BritishAmerican_ret_5d", "AMD_ret_1d", "MRK_Merck_zscore_60d", "AXP_Amex_vol_20d", "EWY_Korea_zscore_60d", "PAYX_Paychex_ret_20d", "spx_vol_5d", "HUM_Humana_ret_5d", "CTAS_Cintas_vol_20d", "heston_var_ev_h7", "PCAR_PaccarInc_ret_5d", "Industrial_Production_zscore_60d", "vix_mean_abs_ret_5d", "US3M_Rate_zscore_60d", "DE_Deere_ret_5d", "AMD_ret_5d"], "is_new": true}, {"model_id": "new_h10_GLOBAL_GradientBoosting_N30_t3", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 10, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "MRK_Merck_zscore_60d", "DOW_Price_zscore_60d", "ORCL_zscore_60d", "PG_ret_20d", "heston_var_ev_h3", "gjr_condvar_h1", "MS_MorganStanley_ret_1d", "DE_Deere_vol_20d", "AMZN_ret_5d", "BDX_Becton_Dickinson_ret_20d", "US3M_Rate_vol_20d", "ASX_Australia_ret_5d", "BLK_BlackRock_zscore_60d", "EWS_Singapore_ret_5d", "MSTR_Bitcoin3_ret_5d", "NWL_Newell_ret_20d", "TM_Telephone_vol_20d", "BA_ret_1d", "ES_Evergy_ret_1d", "QQQ_vol_20d", "FedFunds_zscore_60d", "heston_var_ev_h7", "HD_ret_1d", "HD_zscore_60d", "US3Y_Rate_ret_5d", "Brent_Oil_FRED_ret_20d", "TED_Spread_vol_20d", "SCHW_Schwab_ret_5d"], "is_new": true}, {"model_id": "new_h10_GLOBAL_GradientBoosting_N30_t4", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 10, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "Nikkei_Japan_zscore_60d", "CPB_CampbellSoup_ret_20d", "heston_var_ev_h3", "US30Y_Rate_ret_20d", "EWJ_Japan_vol_20d", "T10Y2Y_Spread_ret_5d", "EWY_Korea_zscore_60d", "FedFunds_zscore_60d", "MSTR_Bitcoin3_ret_1d", "EWQ_France_ret_20d", "PPL_PPL_ret_1d", "NVDA_vol_20d", "MSTR_Bitcoin3_ret_20d", "TXN_vol_20d", "TGT_Target_zscore_60d", "MS_MorganStanley_ret_1d", "EWL_Switzerland_vol_20d", "Core_PCE_zscore_60d", "CCI_CrownCastle_vol_20d", "EOG_EOGResources_vol_20d", "XLB_Materials_zscore_60d", "EXC_Exelon_ret_1d", "HD_zscore_60d", "PCAR_PaccarInc_ret_5d", "PFE_ret_1d", "MO_AltriaMG_ret_1d", "EMR_Emerson_ret_20d", "BDX_Becton_Dickinson_ret_20d"], "is_new": true}, {"model_id": "new_h10_GLOBAL_GradientBoosting_N30_t5", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 10, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "BA_ret_1d", "EWM_Malaysia_zscore_60d", "ES_Evergy_ret_1d", "HD_ret_5d", "DAX_Germany_zscore_60d", "Nikkei_Japan_vol_20d", "HD_ret_20d", "HD_ret_1d", "T10Y2Y_Spread_ret_5d", "SJM_JM_Smucker_ret_1d", "NWL_Newell_ret_20d", "US3M_Rate_vol_20d", "MSTR_Bitcoin3_ret_1d", "heston_var_ev_h5", "Retail_Sales_zscore_60d", "NEE_NextEra_ret_20d", "US1Y_Rate_ret_20d", "EWJ_Japan_vol_20d", "MS_MorganStanley_zscore_60d", "DE_Deere_vol_20d", "NVDA_vol_20d", "HangSeng_HK_ret_5d", "CI_Cigna_vol_20d", "HangSeng_HK_vol_20d", "BTI_BritishAmerican_ret_20d", "EQIX_Equinix_ret_5d", "EXC_Exelon_zscore_60d", "Nikkei_Japan_zscore_60d"], "is_new": true}, {"model_id": "new_h10_GLOBAL_GradientBoosting_N30_t6", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 10, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "LLY_zscore_60d", "TGT_Target_zscore_60d", "VOD_Vodafone_zscore_60d", "EWG_Germany_vol_20d", "TED_Spread_vol_20d", "VRP_ma5", "XOM_ret_1d", "AMD_ret_5d", "DOW_Price_zscore_60d", "MRK_Merck_zscore_60d", "HD_ret_1d", "BLK_BlackRock_zscore_60d", "gjr_condvar_h1", "spx_momentum_3d", "SBUX_vol_20d", "US1Y_Rate_ret_20d", "US3Y_Rate_ret_5d", "IBEX_Spain_ret_20d", "ES_Evergy_ret_1d", "3M_vol_20d", "ORCL_zscore_60d", "M_Macys_vol_20d", "AVB_AvalonBay_zscore_60d", "FedFunds_zscore_60d", "PCAR_PaccarInc_ret_5d", "T10Y2Y_Spread_ret_5d", "EWG_Germany_ret_20d", "Michigan_Sentiment_ret_20d"], "is_new": true}, {"model_id": "new_h10_GLOBAL_GradientBoosting_N30_t7", "algo": "GradientBoosting", "regime": "GLOBAL", "horizon": 10, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "heston_var_ev_h3", "LOW_Lowes_ret_5d", "LUV_SouthwestAir_ret_5d", "EWC_Canada_zscore_60d", "MS_MorganStanley_ret_1d", "EWY_Korea_ret_20d", "Core_PCE_zscore_60d", "SLB_Schlumberger_ret_1d", "DHR_vol_20d", "FedFunds_zscore_60d", "DE_Deere_vol_20d", "LLY_zscore_60d", "TED_Spread_zscore_60d", "AXP_Amex_vol_20d", "spx_vol_5d", "IYM_BasicMaterials_ret_20d", "DIS_vol_20d", "Nikkei_Japan_vol_20d", "Brent_Oil_FRED_ret_20d", "CLX_Clorox_vol_20d", "EWY_Korea_zscore_60d", "EWG_Germany_ret_20d", "EWL_Switzerland_zscore_60d", "PPL_PPL_ret_1d", "ENB_EnbridgeInc_ret_1d", "INTC_ret_5d", "MRK_Merck_zscore_60d", "US5Y_Rate_ret_5d"], "is_new": true}, {"model_id": "new_h10_GLOBAL_RandomForest_N5_t0", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 10, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "MS_MorganStanley_ret_1d", "XOM_ret_1d", "QQQ_vol_20d"], "is_new": true}, {"model_id": "new_h10_GLOBAL_RandomForest_N5_t1", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 10, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "PG_ret_20d", "EWM_Malaysia_vol_20d", "EXC_Exelon_ret_1d"], "is_new": true}, {"model_id": "new_h10_GLOBAL_RandomForest_N5_t2", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 10, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "Brent_Oil_FRED_ret_20d", "MO_AltriaMG_ret_1d", "BDX_Becton_Dickinson_ret_20d"], "is_new": true}, {"model_id": "new_h10_GLOBAL_RandomForest_N5_t3", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 10, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "TM_Telephone_vol_20d", "DAX_Germany_zscore_60d", "AMD_ret_5d"], "is_new": true}, {"model_id": "new_h10_GLOBAL_RandomForest_N5_t4", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 10, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "IYM_BasicMaterials_ret_20d", "HD_ret_1d", "MSTR_Bitcoin3_ret_5d"], "is_new": true}, {"model_id": "new_h10_GLOBAL_RandomForest_N5_t5", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 10, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "Nikkei_Japan_vol_20d", "VVIX_ret_20d", "LMT_LockheedMartin_ret_1d"], "is_new": true}, {"model_id": "new_h10_GLOBAL_RandomForest_N5_t6", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 10, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "Retail_Sales_zscore_60d", "IYM_BasicMaterials_ret_20d", "US3M_Rate_zscore_60d"], "is_new": true}, {"model_id": "new_h10_GLOBAL_RandomForest_N5_t7", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 10, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "Brent_Oil_FRED_ret_5d", "TM_Telephone_ret_1d", "IYR_US_REIT2_zscore_60d"], "is_new": true}, {"model_id": "new_h10_GLOBAL_RandomForest_N8_t0", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 10, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "MS_MorganStanley_ret_1d", "EWJ_Japan_vol_20d", "PFE_ret_1d", "XLK_Tech_zscore_60d", "SCHW_Schwab_ret_5d", "ITT_ITTInc_ret_5d"], "is_new": true}, {"model_id": "new_h10_GLOBAL_RandomForest_N8_t1", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 10, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EMR_Emerson_ret_20d", "PG_ret_20d", "XLB_Materials_zscore_60d", "HD_zscore_60d", "BDX_Becton_Dickinson_ret_20d", "XLF_Fin_vol_20d"], "is_new": true}, {"model_id": "new_h10_GLOBAL_RandomForest_N8_t2", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 10, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "TM_Telephone_vol_20d", "EWL_Switzerland_vol_20d", "MSTR_Bitcoin3_ret_1d", "EOG_EOGResources_ret_5d", "SBUX_zscore_60d", "MO_AltriaMG_ret_1d"], "is_new": true}, {"model_id": "new_h10_GLOBAL_RandomForest_N8_t3", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 10, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWY_Korea_ret_20d", "XLF_Fin_vol_20d", "PFE_ret_1d", "EFFR_vol_20d", "PAYX_Paychex_vol_20d", "SBUX_zscore_60d"], "is_new": true}, {"model_id": "new_h10_GLOBAL_RandomForest_N8_t4", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 10, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "ORCL_zscore_60d", "EOG_EOGResources_ret_5d", "MS_MorganStanley_ret_5d", "CMCSA_ret_1d", "US5Y_Rate_ret_5d", "GE_ret_1d"], "is_new": true}, {"model_id": "new_h10_GLOBAL_RandomForest_N8_t5", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 10, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "MSTR_Bitcoin3_ret_1d", "EWY_Korea_ret_20d", "PAYX_Paychex_zscore_60d", "AVB_AvalonBay_zscore_60d", "PCAR_PaccarInc_ret_5d", "vix_acceleration_1d"], "is_new": true}, {"model_id": "new_h10_GLOBAL_RandomForest_N8_t6", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 10, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "Michigan_Sentiment_ret_20d", "CPB_CampbellSoup_vol_20d", "SLB_Schlumberger_ret_1d", "DE_Deere_vol_20d", "AMD_ret_5d", "VOD_Vodafone_zscore_60d"], "is_new": true}, {"model_id": "new_h10_GLOBAL_RandomForest_N8_t7", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 10, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "PCAR_PaccarInc_ret_5d", "SCHW_Schwab_ret_5d", "PG_ret_20d", "T_ret_1d", "hmm_p_stress", "CI_Cigna_vol_20d"], "is_new": true}, {"model_id": "new_h10_GLOBAL_RandomForest_N10_t0", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 10, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "Brent_Oil_FRED_ret_20d", "ES_Evergy_ret_1d", "EWL_Switzerland_zscore_60d", "TGT_Target_zscore_60d", "spx_momentum_3d", "EWQ_France_ret_20d", "US6M_Rate_ret_20d", "CPB_CampbellSoup_vol_20d"], "is_new": true}, {"model_id": "new_h10_GLOBAL_RandomForest_N10_t1", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 10, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "M_Macys_vol_20d", "CPB_CampbellSoup_ret_5d", "XOM_ret_1d", "XOM_ret_20d", "US7Y_Rate_ret_20d", "AXP_Amex_vol_20d", "IYM_BasicMaterials_ret_20d", "BLK_BlackRock_zscore_60d"], "is_new": true}, {"model_id": "new_h10_GLOBAL_RandomForest_N10_t2", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 10, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "IBEX_Spain_ret_20d", "EQR_Equity_ret_1d", "PAYX_Paychex_ret_20d", "SBUX_zscore_60d", "GILD_Gilead_ret_20d", "EMR_Emerson_ret_20d", "Brent_Oil_FRED_ret_5d", "LMT_LockheedMartin_ret_1d"], "is_new": true}, {"model_id": "new_h10_GLOBAL_RandomForest_N10_t3", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 10, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "PCAR_PaccarInc_ret_5d", "EFFR_vol_20d", "BTI_BritishAmerican_ret_20d", "M_Macys_vol_20d", "AXP_Amex_vol_20d", "CCI_CrownCastle_vol_20d", "SO_SouthernCo_ret_5d", "BLK_BlackRock_zscore_60d"], "is_new": true}, {"model_id": "new_h10_GLOBAL_RandomForest_N10_t4", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 10, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWL_Switzerland_zscore_60d", "ITT_ITTInc_ret_5d", "HD_ret_5d", "SJM_JM_Smucker_ret_5d", "DHR_vol_20d", "spx_abs_ret_max_5d", "EWH_HongKong_ret_5d", "NEE_NextEra_ret_20d"], "is_new": true}, {"model_id": "new_h10_GLOBAL_RandomForest_N10_t5", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 10, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "US3Y_Rate_ret_5d", "M_Macys_vol_20d", "EWY_Korea_zscore_60d", "DHR_vol_20d", "LLY_zscore_60d", "XLV_Health_zscore_60d", "SLB_Schlumberger_ret_5d", "EWL_Switzerland_zscore_60d"], "is_new": true}, {"model_id": "new_h10_GLOBAL_RandomForest_N10_t6", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 10, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "QQQ_vol_20d", "ITT_ITTInc_ret_5d", "MSTR_Bitcoin3_ret_20d", "EWG_Germany_ret_20d", "SBUX_ret_5d", "Industrial_Production_zscore_60d", "HD_zscore_60d", "AXP_Amex_ret_20d"], "is_new": true}, {"model_id": "new_h10_GLOBAL_RandomForest_N10_t7", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 10, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWY_Korea_ret_20d", "EXC_Exelon_ret_1d", "MRK_Merck_zscore_60d", "HangSeng_HK_vol_20d", "INTC_ret_1d", "NFCI_ret_5d", "INTC_ret_5d", "CCI_CrownCastle_vol_20d"], "is_new": true}, {"model_id": "new_h10_GLOBAL_RandomForest_N12_t0", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 10, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "IYM_BasicMaterials_ret_20d", "TED_Spread_zscore_60d", "LLY_zscore_60d", "EWY_Korea_ret_20d", "EWG_Germany_ret_20d", "GE_ret_1d", "HangSeng_HK_ret_1d", "spx_abs_ret_max_5d", "CPB_CampbellSoup_ret_20d", "EQR_Equity_ret_1d"], "is_new": true}, {"model_id": "new_h10_GLOBAL_RandomForest_N12_t1", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 10, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "CPB_CampbellSoup_ret_5d", "US3M_Rate_vol_20d", "US7Y_Rate_ret_20d", "EWJ_Japan_vol_20d", "ORCL_zscore_60d", "AMT_AmericanTower_ret_1d", "TM_Telephone_ret_1d", "Michigan_Sentiment_ret_20d", "FedFunds_zscore_60d", "hmm_p_stress"], "is_new": true}, {"model_id": "new_h10_GLOBAL_RandomForest_N12_t2", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 10, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "QQQ_vol_20d", "AXP_Amex_vol_20d", "Nikkei_Japan_zscore_60d", "Industrial_Production_zscore_60d", "3M_vol_20d", "TGT_Target_zscore_60d", "ITT_ITTInc_ret_5d", "US3M_Rate_vol_20d", "spx_momentum_3d", "T_ret_1d"], "is_new": true}, {"model_id": "new_h10_GLOBAL_RandomForest_N12_t3", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 10, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "US30Y_Rate_ret_20d", "CI_Cigna_vol_20d", "FedFunds_zscore_60d", "ITT_ITTInc_ret_5d", "DAX_Germany_zscore_60d", "SJM_JM_Smucker_ret_1d", "EWM_Malaysia_vol_20d", "Brent_Oil_FRED_ret_20d", "US1Y_Rate_ret_20d", "ES_Evergy_ret_1d"], "is_new": true}, {"model_id": "new_h10_GLOBAL_RandomForest_N12_t4", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 10, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "heston_var_ev_h3", "HD_ret_1d", "LOW_Lowes_ret_20d", "EXC_Exelon_ret_1d", "DE_Deere_ret_5d", "CPB_CampbellSoup_ret_20d", "CI_Cigna_vol_20d", "Nikkei_Japan_vol_20d", "FedFunds_zscore_60d", "NOC_Northrop_ret_20d"], "is_new": true}, {"model_id": "new_h10_GLOBAL_RandomForest_N12_t5", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 10, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "MS_MorganStanley_ret_1d", "EWM_Malaysia_zscore_60d", "EWL_Switzerland_zscore_60d", "VRP_ma5", "BTI_BritishAmerican_ret_20d", "DE_Deere_vol_20d", "Michigan_Sentiment_ret_20d", "JNJ_ret_1d", "PFE_ret_1d", "SPY_zscore_60d"], "is_new": true}, {"model_id": "new_h10_GLOBAL_RandomForest_N12_t6", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 10, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "heston_var_ev_h5", "Core_CPI_zscore_60d", "EFFR_vol_20d", "HangSeng_HK_vol_20d", "3M_ret_5d", "CPB_CampbellSoup_ret_20d", "CMCSA_ret_1d", "Core_PCE_zscore_60d", "SCHW_Schwab_ret_5d", "DAX_Germany_vol_20d"], "is_new": true}, {"model_id": "new_h10_GLOBAL_RandomForest_N12_t7", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 10, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "ORCL_zscore_60d", "ORCL_vol_20d", "Brent_Oil_FRED_ret_5d", "TED_Spread_vol_20d", "WTI_Oil_FRED_zscore_60d", "EWM_Malaysia_zscore_60d", "Retail_Sales_zscore_60d", "INTC_ret_5d", "XOM_ret_1d", "SLB_Schlumberger_ret_1d"], "is_new": true}, {"model_id": "new_h10_GLOBAL_RandomForest_N15_t0", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 10, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "ENB_EnbridgeInc_ret_1d", "EWS_Singapore_ret_5d", "CTAS_Cintas_vol_20d", "DHR_ret_1d", "XLV_Health_zscore_60d", "US3M_Rate_vol_20d", "US5Y_Rate_ret_5d", "EWA_Australia_ret_1d", "NFCI_ret_5d", "PAYX_Paychex_vol_20d", "LLY_zscore_60d", "vix_mean_abs_ret_5d", "XLB_Materials_zscore_60d"], "is_new": true}, {"model_id": "new_h10_GLOBAL_RandomForest_N15_t1", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 10, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "MS_MorganStanley_ret_1d", "EWJ_Japan_vol_20d", "PCAR_PaccarInc_ret_5d", "VOD_Vodafone_zscore_60d", "CPB_CampbellSoup_zscore_60d", "EXC_Exelon_zscore_60d", "BA_ret_1d", "Core_CPI_zscore_60d", "EOG_EOGResources_vol_20d", "ORCL_zscore_60d", "SLB_Schlumberger_ret_1d", "HD_ret_1d", "CPB_CampbellSoup_vol_20d"], "is_new": true}, {"model_id": "new_h10_GLOBAL_RandomForest_N15_t2", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 10, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "hmm_p_stress", "heston_var_ev_h5", "XLV_Health_zscore_60d", "IYR_US_REIT2_zscore_60d", "LUV_SouthwestAir_ret_5d", "NOC_Northrop_ret_20d", "EWH_HongKong_ret_5d", "CMCSA_ret_1d", "EWJ_Japan_vol_20d", "LLY_zscore_60d", "BTI_BritishAmerican_ret_20d", "WTI_Oil_FRED_zscore_60d", "EWM_Malaysia_vol_20d"], "is_new": true}, {"model_id": "new_h10_GLOBAL_RandomForest_N15_t3", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 10, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EOG_EOGResources_ret_5d", "AXP_Amex_vol_20d", "gjr_condvar_h1", "PCAR_PaccarInc_ret_5d", "hmm_p_stress", "DOW_Price_zscore_60d", "HD_ret_20d", "AXP_Amex_ret_20d", "XOM_ret_20d", "M_Macys_vol_20d", "EXC_Exelon_zscore_60d", "US3M_Rate_zscore_60d", "Michigan_Sentiment_ret_20d"], "is_new": true}, {"model_id": "new_h10_GLOBAL_RandomForest_N15_t4", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 10, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "heston_var_ev_h3", "Core_PCE_zscore_60d", "PAYX_Paychex_zscore_60d", "CMCSA_ret_1d", "EOG_EOGResources_ret_5d", "Industrial_Production_zscore_60d", "CI_Cigna_vol_20d", "heston_var_ev_h7", "T10Y2Y_Spread_ret_5d", "CTAS_Cintas_vol_20d", "vix_acceleration_1d", "CPB_CampbellSoup_zscore_60d", "VVIX_ret_20d"], "is_new": true}, {"model_id": "new_h10_GLOBAL_RandomForest_N15_t5", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 10, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "TXN_vol_20d", "MS_MorganStanley_ret_1d", "DE_Deere_vol_20d", "HangSeng_HK_ret_1d", "EWQ_France_ret_20d", "CMCSA_ret_1d", "EWA_Australia_ret_1d", "SLB_Schlumberger_ret_5d", "hmm_p_stress", "PAYX_Paychex_zscore_60d", "EOG_EOGResources_ret_5d", "CPB_CampbellSoup_ret_20d", "MSTR_Bitcoin3_ret_20d"], "is_new": true}, {"model_id": "new_h10_GLOBAL_RandomForest_N15_t6", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 10, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "AXP_Amex_vol_20d", "3M_ret_5d", "VVIX_ret_20d", "EFFR_ret_1d", "CLX_Clorox_vol_20d", "BTI_BritishAmerican_ret_20d", "AMD_ret_5d", "Brent_Oil_FRED_ret_20d", "US5Y_Rate_ret_5d", "EWG_Germany_ret_20d", "spx_momentum_3d", "US1Y_Rate_ret_5d", "EWH_HongKong_ret_5d"], "is_new": true}, {"model_id": "new_h10_GLOBAL_RandomForest_N15_t7", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 10, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWG_Germany_ret_20d", "TED_Spread_vol_20d", "XLY_Disc_vol_20d", "NFCI_ret_5d", "PPL_PPL_ret_1d", "CPB_CampbellSoup_zscore_60d", "EXC_Exelon_ret_1d", "DHR_vol_20d", "IYR_US_REIT2_zscore_60d", "HangSeng_HK_vol_20d", "SJM_JM_Smucker_ret_1d", "BA_ret_1d", "hmm_p_stress"], "is_new": true}, {"model_id": "new_h10_GLOBAL_RandomForest_N20_t0", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 10, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "BLK_BlackRock_zscore_60d", "US30Y_Rate_ret_20d", "TED_Spread_zscore_60d", "MS_MorganStanley_ret_1d", "TM_Telephone_ret_1d", "AMZN_ret_5d", "CPB_CampbellSoup_vol_20d", "heston_ev_h3", "BTI_BritishAmerican_ret_20d", "AMT_AmericanTower_ret_1d", "PAYX_Paychex_vol_20d", "EWQ_France_zscore_60d", "PLD_Prologis_ret_5d", "US6M_Rate_ret_20d", "XLV_Health_zscore_60d", "MS_MorganStanley_zscore_60d", "GE_ret_1d", "VOD_Vodafone_zscore_60d"], "is_new": true}, {"model_id": "new_h10_GLOBAL_RandomForest_N20_t1", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 10, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "XLY_Disc_vol_20d", "Michigan_Sentiment_ret_20d", "US30Y_Rate_ret_20d", "XOM_ret_1d", "US1Y_Rate_ret_5d", "EOG_EOGResources_ret_5d", "TED_Spread_zscore_60d", "NVDA_vol_20d", "EMR_Emerson_ret_20d", "TGT_Target_zscore_60d", "PG_ret_20d", "VRP_ma5", "MRK_Merck_zscore_60d", "MS_MorganStanley_zscore_60d", "EWQ_France_ret_20d", "DE_Deere_ret_5d", "EWS_Singapore_ret_5d", "US3M_Rate_zscore_60d"], "is_new": true}, {"model_id": "new_h10_GLOBAL_RandomForest_N20_t2", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 10, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "DE_Deere_vol_20d", "Brent_Oil_FRED_ret_20d", "spx_vol_5d", "CTAS_Cintas_vol_20d", "HD_zscore_60d", "VVIX_ret_20d", "DE_Deere_ret_5d", "EQR_Equity_ret_1d", "GILD_Gilead_ret_20d", "EFFR_ret_1d", "XLY_Disc_vol_20d", "SPY_zscore_60d", "CMCSA_ret_1d", "CI_Cigna_vol_20d", "US3M_Rate_vol_20d", "XOM_ret_20d", "GE_ret_1d", "EOG_EOGResources_ret_5d"], "is_new": true}, {"model_id": "new_h10_GLOBAL_RandomForest_N20_t3", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 10, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "3M_ret_5d", "NWL_Newell_ret_20d", "DIS_vol_20d", "AMGN_Amgen_ret_1d", "HD_ret_20d", "spx_vol_5d", "ENB_EnbridgeInc_ret_1d", "LMT_LockheedMartin_vol_20d", "CLX_Clorox_vol_20d", "SBUX_ret_5d", "HD_zscore_60d", "GD_GeneralDynamics_zscore_60d", "MSTR_Bitcoin3_ret_20d", "EWQ_France_zscore_60d", "T10Y2Y_Spread_ret_5d", "heston_var_ev_h3", "VOD_Vodafone_zscore_60d", "EQIX_Equinix_ret_5d"], "is_new": true}, {"model_id": "new_h10_GLOBAL_RandomForest_N20_t4", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 10, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "NEE_NextEra_ret_20d", "SJM_JM_Smucker_ret_1d", "SLB_Schlumberger_ret_1d", "US30Y_Rate_ret_20d", "US3Y_Rate_ret_5d", "LLY_zscore_60d", "Industrial_Production_zscore_60d", "EWM_Malaysia_ret_1d", "T10Y2Y_Spread_ret_5d", "EWL_Switzerland_zscore_60d", "EWM_Malaysia_zscore_60d", "HangSeng_HK_vol_20d", "EQR_Equity_ret_1d", "EMR_Emerson_ret_20d", "EWQ_France_ret_20d", "EQIX_Equinix_ret_5d", "BLK_BlackRock_zscore_60d", "GD_GeneralDynamics_zscore_60d"], "is_new": true}, {"model_id": "new_h10_GLOBAL_RandomForest_N20_t5", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 10, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "LOW_Lowes_ret_5d", "EQR_Equity_ret_1d", "HangSeng_HK_ret_5d", "US5Y_Rate_ret_5d", "3M_ret_5d", "GD_GeneralDynamics_zscore_60d", "EWJ_Japan_vol_20d", "spx_vol_5d", "US6M_Rate_ret_20d", "HD_zscore_60d", "SBUX_ret_5d", "WTI_Oil_FRED_zscore_60d", "US3M_Rate_vol_20d", "TM_Telephone_ret_1d", "EOG_EOGResources_ret_5d", "US1Y_Rate_ret_20d", "US1Y_Rate_ret_5d", "MSTR_Bitcoin3_ret_20d"], "is_new": true}, {"model_id": "new_h10_GLOBAL_RandomForest_N20_t6", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 10, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "SJM_JM_Smucker_ret_5d", "PPL_PPL_ret_1d", "GD_GeneralDynamics_zscore_60d", "EOG_EOGResources_vol_20d", "XOM_ret_20d", "VOD_Vodafone_zscore_60d", "TXN_vol_20d", "AMT_AmericanTower_ret_1d", "CTAS_Cintas_vol_20d", "EXC_Exelon_ret_1d", "T_ret_1d", "CPB_CampbellSoup_ret_20d", "EWG_Germany_ret_20d", "QQQ_vol_20d", "DOW_Price_zscore_60d", "XLY_Disc_vol_20d", "EWG_Germany_vol_20d", "EFFR_ret_1d"], "is_new": true}, {"model_id": "new_h10_GLOBAL_RandomForest_N20_t7", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 10, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "DHR_ret_1d", "CPB_CampbellSoup_vol_20d", "EWG_Germany_vol_20d", "ORCL_zscore_60d", "SLB_Schlumberger_ret_5d", "SCHW_Schwab_ret_5d", "ASX_Australia_ret_5d", "vix_mean_abs_ret_5d", "EWA_Australia_zscore_60d", "DAX_Germany_vol_20d", "EWC_Canada_zscore_60d", "XLF_Fin_vol_20d", "spx_momentum_3d", "SJM_JM_Smucker_ret_5d", "Core_PCE_zscore_60d", "M_Macys_vol_20d", "SJM_JM_Smucker_ret_1d", "EWM_Malaysia_ret_1d"], "is_new": true}, {"model_id": "new_h10_GLOBAL_RandomForest_N25_t0", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 10, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "US1Y_Rate_ret_5d", "AXP_Amex_ret_20d", "HUM_Humana_ret_5d", "GE_ret_1d", "SPY_zscore_60d", "SJM_JM_Smucker_ret_1d", "FedFunds_zscore_60d", "DAX_Germany_vol_20d", "TED_Spread_vol_20d", "ORCL_vol_20d", "INTC_ret_1d", "EMR_Emerson_ret_20d", "ASX_Australia_ret_5d", "SLB_Schlumberger_ret_1d", "EWG_Germany_vol_20d", "Industrial_Production_zscore_60d", "US1Y_Rate_ret_20d", "CI_Cigna_vol_20d", "ES_Evergy_ret_1d", "DE_Deere_ret_5d", "WTI_Oil_FRED_zscore_60d", "MS_MorganStanley_zscore_60d", "vix_mean_abs_ret_5d"], "is_new": true}, {"model_id": "new_h10_GLOBAL_RandomForest_N25_t1", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 10, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "3M_ret_5d", "Industrial_Production_zscore_60d", "MSTR_Bitcoin3_ret_20d", "AMT_AmericanTower_ret_1d", "MS_MorganStanley_zscore_60d", "T10Y2Y_Spread_ret_5d", "MSTR_Bitcoin3_ret_5d", "EXC_Exelon_zscore_60d", "EQIX_Equinix_ret_5d", "EWM_Malaysia_zscore_60d", "IBEX_Spain_ret_20d", "SLB_Schlumberger_ret_5d", "WTI_Oil_FRED_zscore_60d", "DIS_vol_20d", "EWQ_France_zscore_60d", "EWC_Canada_zscore_60d", "INTC_ret_1d", "SPY_zscore_60d", "MS_MorganStanley_ret_5d", "XOM_ret_1d", "NOC_Northrop_ret_20d", "HD_ret_5d", "heston_var_ev_h5"], "is_new": true}, {"model_id": "new_h10_GLOBAL_RandomForest_N25_t2", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 10, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "SLB_Schlumberger_ret_5d", "CPB_CampbellSoup_ret_20d", "US5Y_Rate_ret_5d", "IWM_SmallCap_vol_20d", "T_ret_1d", "WTI_Oil_FRED_zscore_60d", "Nikkei_Japan_zscore_60d", "MS_MorganStanley_ret_1d", "SBUX_zscore_60d", "MSTR_Bitcoin3_ret_1d", "M_Macys_vol_20d", "NVDA_vol_20d", "HD_ret_1d", "Core_CPI_zscore_60d", "EWM_Malaysia_vol_20d", "NEE_NextEra_ret_20d", "PPL_PPL_ret_1d", "heston_var_ev_h3", "DHR_ret_1d", "NOC_Northrop_ret_20d", "TGT_Target_zscore_60d", "EFFR_vol_20d", "US3M_Rate_zscore_60d"], "is_new": true}, {"model_id": "new_h10_GLOBAL_RandomForest_N25_t3", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 10, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "GE_ret_1d", "AVB_AvalonBay_zscore_60d", "heston_var_ev_h7", "MSTR_Bitcoin3_ret_20d", "IWM_SmallCap_vol_20d", "LMT_LockheedMartin_ret_1d", "XLV_Health_zscore_60d", "Michigan_Sentiment_ret_20d", "SBUX_zscore_60d", "DE_Deere_ret_5d", "PAYX_Paychex_vol_20d", "heston_var_ev_h5", "US5Y_Rate_ret_5d", "CI_Cigna_vol_20d", "VRP_ma5", "EQIX_Equinix_ret_5d", "DAX_Germany_zscore_60d", "spx_vol_5d", "LOW_Lowes_ret_20d", "FedFunds_zscore_60d", "XLK_Tech_zscore_60d", "US1Y_Rate_ret_5d", "ASX_Australia_ret_5d"], "is_new": true}, {"model_id": "new_h10_GLOBAL_RandomForest_N25_t4", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 10, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "T10Y2Y_Spread_ret_5d", "Industrial_Production_zscore_60d", "GD_GeneralDynamics_zscore_60d", "LLY_zscore_60d", "LUV_SouthwestAir_ret_5d", "AMT_AmericanTower_ret_1d", "XLB_Materials_zscore_60d", "EFFR_vol_20d", "EMR_Emerson_ret_20d", "EWG_Germany_ret_20d", "ASX_Australia_ret_5d", "MSTR_Bitcoin3_ret_1d", "EWS_Singapore_ret_5d", "NVDA_vol_20d", "gjr_condvar_h1", "HangSeng_HK_ret_5d", "CPB_CampbellSoup_ret_5d", "HD_zscore_60d", "XLK_Tech_zscore_60d", "US6M_Rate_ret_20d", "ENB_EnbridgeInc_ret_1d", "AXP_Amex_ret_20d", "XLV_Health_zscore_60d"], "is_new": true}, {"model_id": "new_h10_GLOBAL_RandomForest_N25_t5", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 10, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWM_Malaysia_vol_20d", "US3M_Rate_vol_20d", "EMR_Emerson_ret_20d", "heston_var_ev_h3", "IYR_US_REIT2_zscore_60d", "EWM_Malaysia_ret_1d", "Nikkei_Japan_vol_20d", "heston_var_ev_h5", "EWJ_Japan_vol_20d", "VRP_ma5", "EOG_EOGResources_vol_20d", "EFFR_vol_20d", "SPY_zscore_60d", "heston_var_ev_h7", "ES_Evergy_ret_1d", "T_ret_1d", "TM_Telephone_vol_20d", "ITT_ITTInc_ret_5d", "TGT_Target_zscore_60d", "spx_abs_ret_max_5d", "hmm_p_stress", "GD_GeneralDynamics_zscore_60d", "HangSeng_HK_ret_1d"], "is_new": true}, {"model_id": "new_h10_GLOBAL_RandomForest_N25_t6", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 10, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "INTC_ret_1d", "DIS_vol_20d", "VOD_Vodafone_zscore_60d", "ORCL_vol_20d", "NVDA_vol_20d", "GILD_Gilead_ret_20d", "CI_Cigna_vol_20d", "HD_ret_20d", "MSTR_Bitcoin3_ret_20d", "PG_ret_20d", "HangSeng_HK_vol_20d", "CPB_CampbellSoup_zscore_60d", "CPB_CampbellSoup_ret_20d", "TXN_vol_20d", "Brent_Oil_FRED_ret_5d", "LMT_LockheedMartin_vol_20d", "MSTR_Bitcoin3_ret_5d", "US1Y_Rate_ret_20d", "M_Macys_vol_20d", "CTAS_Cintas_vol_20d", "IWM_SmallCap_vol_20d", "EWG_Germany_vol_20d", "gjr_condvar_h1"], "is_new": true}, {"model_id": "new_h10_GLOBAL_RandomForest_N25_t7", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 10, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWQ_France_zscore_60d", "XLB_Materials_zscore_60d", "XLV_Health_zscore_60d", "PAYX_Paychex_ret_20d", "DAX_Germany_zscore_60d", "spx_vol_5d", "EWL_Switzerland_vol_20d", "INTC_ret_5d", "MSTR_Bitcoin3_ret_20d", "ORCL_zscore_60d", "Nikkei_Japan_zscore_60d", "TM_Telephone_vol_20d", "GILD_Gilead_ret_20d", "AVB_AvalonBay_zscore_60d", "EWG_Germany_vol_20d", "Retail_Sales_zscore_60d", "CCI_CrownCastle_vol_20d", "GE_ret_1d", "CI_Cigna_vol_20d", "SJM_JM_Smucker_ret_1d", "US3M_Rate_vol_20d", "AMT_AmericanTower_ret_1d", "VVIX_ret_20d"], "is_new": true}, {"model_id": "new_h10_GLOBAL_RandomForest_N30_t0", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 10, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWM_Malaysia_ret_1d", "EWC_Canada_zscore_60d", "TXN_vol_20d", "T10Y2Y_Spread_ret_5d", "TGT_Target_zscore_60d", "SCHW_Schwab_ret_5d", "AXP_Amex_ret_20d", "HangSeng_HK_ret_1d", "BLK_BlackRock_zscore_60d", "Nikkei_Japan_zscore_60d", "BTI_BritishAmerican_ret_20d", "AMGN_Amgen_ret_1d", "EFFR_vol_20d", "XLB_Materials_zscore_60d", "spx_vol_5d", "EQIX_Equinix_ret_5d", "BTI_BritishAmerican_ret_5d", "SBUX_zscore_60d", "ORCL_zscore_60d", "SBUX_ret_5d", "heston_ev_h3", "IWM_SmallCap_vol_20d", "ORCL_vol_20d", "GILD_Gilead_ret_20d", "EWY_Korea_zscore_60d", "VVIX_ret_20d", "EWA_Australia_ret_1d", "PAYX_Paychex_ret_20d"], "is_new": true}, {"model_id": "new_h10_GLOBAL_RandomForest_N30_t1", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 10, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "MSTR_Bitcoin3_ret_5d", "NFCI_ret_5d", "AXP_Amex_ret_20d", "vix_acceleration_1d", "hmm_p_stress", "EXC_Exelon_ret_1d", "AXP_Amex_vol_20d", "EXC_Exelon_zscore_60d", "CTAS_Cintas_vol_20d", "LMT_LockheedMartin_ret_1d", "EWL_Switzerland_vol_20d", "Nikkei_Japan_vol_20d", "LMT_LockheedMartin_vol_20d", "VVIX_ret_20d", "AMZN_ret_5d", "HUM_Humana_ret_5d", "AMT_AmericanTower_ret_1d", "QQQ_vol_20d", "CPB_CampbellSoup_zscore_60d", "LUV_SouthwestAir_ret_5d", "HangSeng_HK_ret_5d", "EFFR_vol_20d", "IWM_SmallCap_vol_20d", "BTI_BritishAmerican_ret_20d", "SLB_Schlumberger_ret_1d", "ORCL_zscore_60d", "Core_CPI_zscore_60d", "HD_zscore_60d"], "is_new": true}, {"model_id": "new_h10_GLOBAL_RandomForest_N30_t2", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 10, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "VRP_ma5", "DHR_ret_1d", "vix_mean_abs_ret_5d", "XLK_Tech_zscore_60d", "MSTR_Bitcoin3_ret_20d", "Brent_Oil_FRED_ret_5d", "EWQ_France_zscore_60d", "CLX_Clorox_vol_20d", "heston_var_ev_h7", "SJM_JM_Smucker_ret_1d", "CI_Cigna_vol_20d", "BA_ret_1d", "US3M_Rate_zscore_60d", "Core_PCE_zscore_60d", "BLK_BlackRock_zscore_60d", "QQQ_vol_20d", "ITT_ITTInc_ret_5d", "MSTR_Bitcoin3_ret_1d", "EFFR_ret_1d", "GD_GeneralDynamics_zscore_60d", "EOG_EOGResources_vol_20d", "PLD_Prologis_ret_5d", "LLY_zscore_60d", "ASX_Australia_vol_20d", "EMR_Emerson_ret_20d", "HD_zscore_60d", "TM_Telephone_ret_1d", "SBUX_ret_5d"], "is_new": true}, {"model_id": "new_h10_GLOBAL_RandomForest_N30_t3", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 10, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "TED_Spread_zscore_60d", "XOM_ret_20d", "DE_Deere_ret_5d", "PAYX_Paychex_ret_20d", "WTI_Oil_FRED_zscore_60d", "EWL_Switzerland_vol_20d", "GD_GeneralDynamics_zscore_60d", "NOC_Northrop_ret_20d", "INTC_ret_1d", "ORCL_vol_20d", "IBEX_Spain_ret_20d", "CCI_CrownCastle_vol_20d", "EFFR_ret_1d", "EQIX_Equinix_ret_5d", "CMCSA_ret_1d", "Michigan_Sentiment_ret_20d", "LMT_LockheedMartin_ret_1d", "EWC_Canada_zscore_60d", "EWM_Malaysia_zscore_60d", "NFCI_ret_5d", "EWG_Germany_ret_20d", "AXP_Amex_vol_20d", "IYR_US_REIT2_zscore_60d", "SBUX_zscore_60d", "SJM_JM_Smucker_ret_5d", "IYM_BasicMaterials_ret_20d", "EXC_Exelon_ret_1d", "CPB_CampbellSoup_vol_20d"], "is_new": true}, {"model_id": "new_h10_GLOBAL_RandomForest_N30_t4", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 10, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "AXP_Amex_ret_20d", "INTC_ret_5d", "EWL_Switzerland_zscore_60d", "heston_var_ev_h3", "EWC_Canada_zscore_60d", "EWM_Malaysia_zscore_60d", "US1Y_Rate_ret_5d", "US5Y_Rate_ret_5d", "DAX_Germany_vol_20d", "DAX_Germany_zscore_60d", "CMCSA_ret_1d", "AMD_ret_5d", "HD_zscore_60d", "VRP_ma5", "IBEX_Spain_ret_20d", "AMD_ret_1d", "EWQ_France_ret_20d", "heston_ev_h3", "HD_ret_1d", "TXN_vol_20d", "3M_ret_5d", "BA_ret_1d", "IYM_BasicMaterials_ret_20d", "MS_MorganStanley_zscore_60d", "ES_Evergy_ret_1d", "LLY_zscore_60d", "NEE_NextEra_ret_20d", "CLX_Clorox_vol_20d"], "is_new": true}, {"model_id": "new_h10_GLOBAL_RandomForest_N30_t5", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 10, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWA_Australia_zscore_60d", "EOG_EOGResources_vol_20d", "BTI_BritishAmerican_ret_20d", "PCAR_PaccarInc_ret_5d", "GE_ret_1d", "AORD_AUS_zscore_60d", "NFCI_ret_5d", "JNJ_ret_1d", "PAYX_Paychex_ret_20d", "heston_var_ev_h5", "NOC_Northrop_ret_20d", "CPB_CampbellSoup_zscore_60d", "Michigan_Sentiment_ret_20d", "SCHW_Schwab_ret_5d", "EWL_Switzerland_vol_20d", "DE_Deere_vol_20d", "vix_acceleration_1d", "PFE_ret_1d", "CPB_CampbellSoup_ret_20d", "IWM_SmallCap_vol_20d", "IBEX_Spain_ret_20d", "SJM_JM_Smucker_ret_5d", "HangSeng_HK_vol_20d", "Core_CPI_zscore_60d", "spx_vol_5d", "vix_mean_abs_ret_5d", "AMD_ret_5d", "DHR_ret_1d"], "is_new": true}, {"model_id": "new_h10_GLOBAL_RandomForest_N30_t6", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 10, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "US3M_Rate_zscore_60d", "spx_momentum_3d", "EFFR_ret_1d", "SJM_JM_Smucker_ret_1d", "US1Y_Rate_ret_5d", "SO_SouthernCo_ret_5d", "Core_CPI_zscore_60d", "XLY_Disc_vol_20d", "CPB_CampbellSoup_ret_5d", "DAX_Germany_vol_20d", "JNJ_ret_1d", "INTC_ret_1d", "CPB_CampbellSoup_vol_20d", "XLF_Fin_vol_20d", "DIS_vol_20d", "LLY_zscore_60d", "BA_ret_1d", "VOD_Vodafone_zscore_60d", "CPB_CampbellSoup_ret_20d", "SBUX_zscore_60d", "heston_ev_h3", "LOW_Lowes_ret_5d", "NFCI_ret_5d", "PAYX_Paychex_ret_20d", "vix_mean_abs_ret_5d", "IWM_SmallCap_vol_20d", "WTI_Oil_FRED_zscore_60d", "PAYX_Paychex_vol_20d"], "is_new": true}, {"model_id": "new_h10_GLOBAL_RandomForest_N30_t7", "algo": "RandomForest", "regime": "GLOBAL", "horizon": 10, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "DE_Deere_vol_20d", "MO_AltriaMG_ret_1d", "PPL_PPL_ret_1d", "EWL_Switzerland_vol_20d", "CPB_CampbellSoup_vol_20d", "AORD_AUS_zscore_60d", "AXP_Amex_vol_20d", "XLK_Tech_zscore_60d", "INTC_ret_5d", "IWM_SmallCap_vol_20d", "ASX_Australia_vol_20d", "EWH_HongKong_ret_5d", "HD_ret_5d", "heston_var_ev_h5", "AMT_AmericanTower_ret_1d", "EFFR_vol_20d", "SJM_JM_Smucker_ret_5d", "3M_ret_5d", "VVIX_ret_20d", "INTC_ret_1d", "NFCI_ret_5d", "ORCL_zscore_60d", "EWG_Germany_ret_20d", "NWL_Newell_ret_20d", "EFFR_ret_1d", "Nikkei_Japan_vol_20d", "CPB_CampbellSoup_ret_5d", "Retail_Sales_zscore_60d"], "is_new": true}, {"model_id": "new_h10_GLOBAL_LogisticRegression_N5_t0", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 10, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "Core_CPI_zscore_60d", "NWL_Newell_ret_20d", "XOM_ret_20d"], "is_new": true}, {"model_id": "new_h10_GLOBAL_LogisticRegression_N5_t1", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 10, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EXC_Exelon_ret_1d", "CLX_Clorox_vol_20d", "EWC_Canada_zscore_60d"], "is_new": true}, {"model_id": "new_h10_GLOBAL_LogisticRegression_N5_t2", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 10, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EQR_Equity_ret_1d", "vix_mean_abs_ret_5d", "SBUX_vol_20d"], "is_new": true}, {"model_id": "new_h10_GLOBAL_LogisticRegression_N5_t3", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 10, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "Brent_Oil_FRED_ret_5d", "CI_Cigna_vol_20d", "EWM_Malaysia_zscore_60d"], "is_new": true}, {"model_id": "new_h10_GLOBAL_LogisticRegression_N5_t4", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 10, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EFFR_ret_1d", "GILD_Gilead_ret_20d", "IBEX_Spain_ret_20d"], "is_new": true}, {"model_id": "new_h10_GLOBAL_LogisticRegression_N5_t5", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 10, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWM_Malaysia_ret_1d", "TM_Telephone_ret_1d", "CPB_CampbellSoup_ret_20d"], "is_new": true}, {"model_id": "new_h10_GLOBAL_LogisticRegression_N5_t6", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 10, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "PAYX_Paychex_ret_20d", "gjr_condvar_h1", "BA_ret_1d"], "is_new": true}, {"model_id": "new_h10_GLOBAL_LogisticRegression_N5_t7", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 10, "n_features": 5, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "hmm_p_stress", "CPB_CampbellSoup_ret_20d", "AORD_AUS_zscore_60d"], "is_new": true}, {"model_id": "new_h10_GLOBAL_LogisticRegression_N8_t0", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 10, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "US30Y_Rate_ret_20d", "AMD_ret_5d", "AMZN_ret_5d", "US1Y_Rate_ret_5d", "US7Y_Rate_ret_20d", "AMGN_Amgen_ret_1d"], "is_new": true}, {"model_id": "new_h10_GLOBAL_LogisticRegression_N8_t1", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 10, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "Nikkei_Japan_zscore_60d", "LMT_LockheedMartin_vol_20d", "SCHW_Schwab_ret_5d", "AMD_ret_5d", "TED_Spread_vol_20d", "EXC_Exelon_ret_1d"], "is_new": true}, {"model_id": "new_h10_GLOBAL_LogisticRegression_N8_t2", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 10, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "BA_ret_1d", "LOW_Lowes_ret_20d", "XOM_ret_1d", "spx_abs_ret_max_5d", "heston_ev_h3", "AORD_AUS_zscore_60d"], "is_new": true}, {"model_id": "new_h10_GLOBAL_LogisticRegression_N8_t3", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 10, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "AVB_AvalonBay_zscore_60d", "CPB_CampbellSoup_ret_20d", "US7Y_Rate_ret_20d", "JNJ_ret_1d", "DAX_Germany_vol_20d", "ITT_ITTInc_ret_5d"], "is_new": true}, {"model_id": "new_h10_GLOBAL_LogisticRegression_N8_t4", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 10, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "PPL_PPL_ret_1d", "US3Y_Rate_ret_5d", "MSTR_Bitcoin3_ret_1d", "INTC_ret_5d", "BLK_BlackRock_zscore_60d", "NEE_NextEra_ret_20d"], "is_new": true}, {"model_id": "new_h10_GLOBAL_LogisticRegression_N8_t5", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 10, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "IYR_US_REIT2_zscore_60d", "DE_Deere_vol_20d", "SBUX_ret_5d", "MS_MorganStanley_ret_1d", "WTI_Oil_FRED_zscore_60d", "SCHW_Schwab_ret_5d"], "is_new": true}, {"model_id": "new_h10_GLOBAL_LogisticRegression_N8_t6", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 10, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "spx_momentum_3d", "MRK_Merck_zscore_60d", "HD_ret_1d", "EWM_Malaysia_zscore_60d", "ORCL_vol_20d", "EOG_EOGResources_vol_20d"], "is_new": true}, {"model_id": "new_h10_GLOBAL_LogisticRegression_N8_t7", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 10, "n_features": 8, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "IYR_US_REIT2_zscore_60d", "AORD_AUS_zscore_60d", "FedFunds_zscore_60d", "Core_PCE_zscore_60d", "BLK_BlackRock_zscore_60d", "EWQ_France_ret_20d"], "is_new": true}, {"model_id": "new_h10_GLOBAL_LogisticRegression_N10_t0", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 10, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWM_Malaysia_ret_1d", "EWL_Switzerland_vol_20d", "MO_AltriaMG_ret_1d", "LOW_Lowes_ret_5d", "vix_mean_abs_ret_5d", "CMCSA_ret_1d", "US3M_Rate_vol_20d", "PAYX_Paychex_vol_20d"], "is_new": true}, {"model_id": "new_h10_GLOBAL_LogisticRegression_N10_t1", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 10, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "AXP_Amex_ret_20d", "EWQ_France_ret_20d", "INTC_ret_1d", "SLB_Schlumberger_ret_5d", "CI_Cigna_vol_20d", "ITT_ITTInc_ret_5d", "LOW_Lowes_ret_5d", "EWC_Canada_zscore_60d"], "is_new": true}, {"model_id": "new_h10_GLOBAL_LogisticRegression_N10_t2", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 10, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "ORCL_zscore_60d", "CI_Cigna_vol_20d", "BDX_Becton_Dickinson_ret_20d", "HD_ret_1d", "HUM_Humana_ret_5d", "XLY_Disc_vol_20d", "PCAR_PaccarInc_ret_5d", "US3M_Rate_zscore_60d"], "is_new": true}, {"model_id": "new_h10_GLOBAL_LogisticRegression_N10_t3", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 10, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "LMT_LockheedMartin_ret_1d", "MRK_Merck_zscore_60d", "LOW_Lowes_ret_5d", "EWC_Canada_zscore_60d", "DAX_Germany_zscore_60d", "US3M_Rate_vol_20d", "AXP_Amex_vol_20d", "TXN_vol_20d"], "is_new": true}, {"model_id": "new_h10_GLOBAL_LogisticRegression_N10_t4", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 10, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "DAX_Germany_vol_20d", "EMR_Emerson_ret_20d", "IYM_BasicMaterials_ret_20d", "US30Y_Rate_ret_20d", "Brent_Oil_FRED_ret_20d", "ORCL_vol_20d", "AMD_ret_1d", "IWM_SmallCap_vol_20d"], "is_new": true}, {"model_id": "new_h10_GLOBAL_LogisticRegression_N10_t5", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 10, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "GE_ret_1d", "MS_MorganStanley_zscore_60d", "INTC_ret_5d", "Industrial_Production_zscore_60d", "TGT_Target_zscore_60d", "TM_Telephone_ret_1d", "MSTR_Bitcoin3_ret_5d", "PCAR_PaccarInc_ret_5d"], "is_new": true}, {"model_id": "new_h10_GLOBAL_LogisticRegression_N10_t6", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 10, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "US3Y_Rate_ret_5d", "EWA_Australia_zscore_60d", "GD_GeneralDynamics_zscore_60d", "VOD_Vodafone_zscore_60d", "vix_acceleration_1d", "EQIX_Equinix_ret_5d", "AXP_Amex_ret_20d", "XLV_Health_zscore_60d"], "is_new": true}, {"model_id": "new_h10_GLOBAL_LogisticRegression_N10_t7", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 10, "n_features": 10, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "HangSeng_HK_ret_5d", "EWA_Australia_ret_1d", "HD_ret_5d", "US3Y_Rate_ret_5d", "EWH_HongKong_ret_5d", "INTC_ret_5d", "AVB_AvalonBay_zscore_60d", "EOG_EOGResources_vol_20d"], "is_new": true}, {"model_id": "new_h10_GLOBAL_LogisticRegression_N12_t0", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 10, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "CMCSA_ret_1d", "TED_Spread_vol_20d", "NVDA_vol_20d", "EWQ_France_ret_20d", "M_Macys_vol_20d", "PAYX_Paychex_zscore_60d", "EWG_Germany_vol_20d", "ORCL_zscore_60d", "US3Y_Rate_ret_5d", "EWC_Canada_zscore_60d"], "is_new": true}, {"model_id": "new_h10_GLOBAL_LogisticRegression_N12_t1", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 10, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "SBUX_zscore_60d", "AMGN_Amgen_ret_1d", "XLV_Health_zscore_60d", "AMD_ret_1d", "gjr_condvar_h1", "3M_ret_5d", "HangSeng_HK_ret_5d", "VVIX_ret_20d", "Core_CPI_zscore_60d", "EOG_EOGResources_vol_20d"], "is_new": true}, {"model_id": "new_h10_GLOBAL_LogisticRegression_N12_t2", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 10, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "GE_ret_1d", "US1Y_Rate_ret_5d", "spx_abs_ret_max_5d", "VRP_ma5", "IYM_BasicMaterials_ret_20d", "PLD_Prologis_ret_5d", "T10Y2Y_Spread_ret_5d", "EWY_Korea_ret_20d", "DAX_Germany_vol_20d", "MS_MorganStanley_ret_5d"], "is_new": true}, {"model_id": "new_h10_GLOBAL_LogisticRegression_N12_t3", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 10, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWY_Korea_ret_20d", "3M_vol_20d", "SJM_JM_Smucker_ret_1d", "Industrial_Production_zscore_60d", "EFFR_ret_1d", "ORCL_vol_20d", "JNJ_ret_1d", "SLB_Schlumberger_ret_1d", "DE_Deere_vol_20d", "spx_abs_ret_max_5d"], "is_new": true}, {"model_id": "new_h10_GLOBAL_LogisticRegression_N12_t4", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 10, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EMR_Emerson_ret_20d", "AMT_AmericanTower_ret_1d", "IBEX_Spain_ret_20d", "EWA_Australia_zscore_60d", "EWY_Korea_ret_20d", "ORCL_vol_20d", "DOW_Price_zscore_60d", "DHR_ret_1d", "CPB_CampbellSoup_vol_20d", "GILD_Gilead_ret_20d"], "is_new": true}, {"model_id": "new_h10_GLOBAL_LogisticRegression_N12_t5", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 10, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "VRP_ma5", "T_ret_1d", "SPY_zscore_60d", "EWM_Malaysia_ret_1d", "LLY_zscore_60d", "DAX_Germany_zscore_60d", "AMD_ret_1d", "AMZN_ret_5d", "PG_ret_20d", "TXN_vol_20d"], "is_new": true}, {"model_id": "new_h10_GLOBAL_LogisticRegression_N12_t6", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 10, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWM_Malaysia_ret_1d", "ENB_EnbridgeInc_ret_1d", "CPB_CampbellSoup_ret_20d", "HD_ret_1d", "EWY_Korea_ret_20d", "LOW_Lowes_ret_5d", "gjr_condvar_h1", "IYR_US_REIT2_zscore_60d", "JNJ_ret_1d", "Brent_Oil_FRED_ret_5d"], "is_new": true}, {"model_id": "new_h10_GLOBAL_LogisticRegression_N12_t7", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 10, "n_features": 12, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "gjr_condvar_h1", "NWL_Newell_ret_20d", "SJM_JM_Smucker_ret_1d", "EQIX_Equinix_ret_5d", "Michigan_Sentiment_ret_20d", "EWA_Australia_zscore_60d", "vix_acceleration_1d", "DOW_Price_zscore_60d", "ITT_ITTInc_ret_5d", "BDX_Becton_Dickinson_ret_20d"], "is_new": true}, {"model_id": "new_h10_GLOBAL_LogisticRegression_N15_t0", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 10, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "SBUX_vol_20d", "EQIX_Equinix_ret_5d", "Brent_Oil_FRED_ret_20d", "TED_Spread_vol_20d", "ASX_Australia_ret_5d", "VOD_Vodafone_zscore_60d", "TED_Spread_zscore_60d", "EXC_Exelon_ret_1d", "NVDA_vol_20d", "hmm_p_stress", "WTI_Oil_FRED_zscore_60d", "SBUX_zscore_60d", "EWG_Germany_ret_20d"], "is_new": true}, {"model_id": "new_h10_GLOBAL_LogisticRegression_N15_t1", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 10, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "IYM_BasicMaterials_ret_20d", "NOC_Northrop_ret_20d", "DE_Deere_vol_20d", "M_Macys_vol_20d", "DAX_Germany_zscore_60d", "EWM_Malaysia_vol_20d", "PAYX_Paychex_ret_20d", "MS_MorganStanley_zscore_60d", "EFFR_vol_20d", "US5Y_Rate_ret_5d", "MSTR_Bitcoin3_ret_1d", "ITT_ITTInc_ret_5d", "INTC_ret_5d"], "is_new": true}, {"model_id": "new_h10_GLOBAL_LogisticRegression_N15_t2", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 10, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EFFR_ret_1d", "GE_ret_1d", "EXC_Exelon_zscore_60d", "Core_CPI_zscore_60d", "M_Macys_vol_20d", "LUV_SouthwestAir_ret_5d", "SLB_Schlumberger_ret_1d", "MSTR_Bitcoin3_ret_20d", "EWY_Korea_ret_20d", "HangSeng_HK_vol_20d", "SBUX_zscore_60d", "TM_Telephone_vol_20d", "EQR_Equity_ret_1d"], "is_new": true}, {"model_id": "new_h10_GLOBAL_LogisticRegression_N15_t3", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 10, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "HangSeng_HK_vol_20d", "EWY_Korea_zscore_60d", "EWA_Australia_zscore_60d", "BLK_BlackRock_zscore_60d", "Brent_Oil_FRED_ret_5d", "US1Y_Rate_ret_20d", "SCHW_Schwab_ret_5d", "NOC_Northrop_ret_20d", "EWL_Switzerland_zscore_60d", "M_Macys_vol_20d", "HangSeng_HK_ret_5d", "PFE_ret_1d", "NVDA_vol_20d"], "is_new": true}, {"model_id": "new_h10_GLOBAL_LogisticRegression_N15_t4", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 10, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "SBUX_zscore_60d", "IBEX_Spain_ret_20d", "XOM_ret_20d", "HD_ret_1d", "LLY_zscore_60d", "US3M_Rate_vol_20d", "heston_ev_h3", "EQIX_Equinix_ret_5d", "EWA_Australia_ret_1d", "EWJ_Japan_vol_20d", "EWM_Malaysia_vol_20d", "ORCL_vol_20d", "EWY_Korea_zscore_60d"], "is_new": true}, {"model_id": "new_h10_GLOBAL_LogisticRegression_N15_t5", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 10, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "PCAR_PaccarInc_ret_5d", "3M_ret_5d", "EWY_Korea_ret_20d", "3M_vol_20d", "LLY_zscore_60d", "AMD_ret_1d", "CI_Cigna_vol_20d", "vix_mean_abs_ret_5d", "AVB_AvalonBay_zscore_60d", "MS_MorganStanley_ret_5d", "Brent_Oil_FRED_ret_20d", "CPB_CampbellSoup_ret_5d", "EQIX_Equinix_ret_5d"], "is_new": true}, {"model_id": "new_h10_GLOBAL_LogisticRegression_N15_t6", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 10, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "US3Y_Rate_ret_5d", "PFE_ret_1d", "T10Y2Y_Spread_ret_5d", "CPB_CampbellSoup_ret_20d", "Nikkei_Japan_zscore_60d", "SLB_Schlumberger_ret_5d", "NFCI_ret_5d", "HD_ret_20d", "PLD_Prologis_ret_5d", "DE_Deere_vol_20d", "AXP_Amex_ret_20d", "TXN_vol_20d", "Brent_Oil_FRED_ret_5d"], "is_new": true}, {"model_id": "new_h10_GLOBAL_LogisticRegression_N15_t7", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 10, "n_features": 15, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EWG_Germany_vol_20d", "CMCSA_ret_1d", "LUV_SouthwestAir_ret_5d", "MSTR_Bitcoin3_ret_1d", "SPY_zscore_60d", "PPL_PPL_ret_1d", "HangSeng_HK_ret_5d", "DAX_Germany_zscore_60d", "EWG_Germany_ret_20d", "DHR_ret_1d", "TXN_vol_20d", "Core_CPI_zscore_60d", "DE_Deere_ret_5d"], "is_new": true}, {"model_id": "new_h10_GLOBAL_LogisticRegression_N20_t0", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 10, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "HUM_Humana_ret_5d", "INTC_ret_1d", "MSTR_Bitcoin3_ret_20d", "US3M_Rate_zscore_60d", "HD_ret_20d", "DHR_vol_20d", "LUV_SouthwestAir_ret_5d", "SLB_Schlumberger_ret_1d", "EQR_Equity_ret_1d", "DAX_Germany_zscore_60d", "vix_acceleration_1d", "MSTR_Bitcoin3_ret_1d", "IBEX_Spain_ret_20d", "XLF_Fin_vol_20d", "PLD_Prologis_ret_5d", "Brent_Oil_FRED_ret_5d", "3M_ret_5d", "PAYX_Paychex_vol_20d"], "is_new": true}, {"model_id": "new_h10_GLOBAL_LogisticRegression_N20_t1", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 10, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "PLD_Prologis_ret_5d", "heston_ev_h3", "Core_PCE_zscore_60d", "SO_SouthernCo_ret_5d", "hmm_p_stress", "CPB_CampbellSoup_vol_20d", "LOW_Lowes_ret_5d", "DOW_Price_zscore_60d", "BLK_BlackRock_zscore_60d", "XLB_Materials_zscore_60d", "HD_ret_20d", "Brent_Oil_FRED_ret_20d", "US1Y_Rate_ret_20d", "BTI_BritishAmerican_ret_5d", "EWY_Korea_zscore_60d", "PCAR_PaccarInc_ret_5d", "US1Y_Rate_ret_5d", "NWL_Newell_ret_20d"], "is_new": true}, {"model_id": "new_h10_GLOBAL_LogisticRegression_N20_t2", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 10, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "LOW_Lowes_ret_20d", "EWG_Germany_ret_20d", "AXP_Amex_vol_20d", "NFCI_ret_5d", "TXN_vol_20d", "EOG_EOGResources_vol_20d", "MO_AltriaMG_ret_1d", "Retail_Sales_zscore_60d", "JNJ_ret_1d", "EWM_Malaysia_ret_1d", "EWH_HongKong_ret_5d", "ORCL_vol_20d", "TGT_Target_zscore_60d", "US3M_Rate_zscore_60d", "spx_momentum_3d", "MSTR_Bitcoin3_ret_20d", "DOW_Price_zscore_60d", "CTAS_Cintas_vol_20d"], "is_new": true}, {"model_id": "new_h10_GLOBAL_LogisticRegression_N20_t3", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 10, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EXC_Exelon_ret_1d", "EMR_Emerson_ret_20d", "SBUX_vol_20d", "HUM_Humana_ret_5d", "PPL_PPL_ret_1d", "XLV_Health_zscore_60d", "ORCL_vol_20d", "EWM_Malaysia_zscore_60d", "MS_MorganStanley_zscore_60d", "3M_ret_5d", "spx_momentum_3d", "EQIX_Equinix_ret_5d", "HD_ret_5d", "US3M_Rate_zscore_60d", "EWJ_Japan_vol_20d", "DE_Deere_ret_5d", "TM_Telephone_vol_20d", "DIS_vol_20d"], "is_new": true}, {"model_id": "new_h10_GLOBAL_LogisticRegression_N20_t4", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 10, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EWY_Korea_zscore_60d", "HD_ret_1d", "EWG_Germany_vol_20d", "ASX_Australia_ret_5d", "HD_ret_5d", "US5Y_Rate_ret_5d", "CLX_Clorox_vol_20d", "LOW_Lowes_ret_5d", "MSTR_Bitcoin3_ret_20d", "BLK_BlackRock_zscore_60d", "SO_SouthernCo_ret_5d", "ORCL_zscore_60d", "EQIX_Equinix_ret_5d", "IYR_US_REIT2_zscore_60d", "SBUX_vol_20d", "MO_AltriaMG_ret_1d", "LOW_Lowes_ret_20d", "AORD_AUS_zscore_60d"], "is_new": true}, {"model_id": "new_h10_GLOBAL_LogisticRegression_N20_t5", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 10, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "GD_GeneralDynamics_zscore_60d", "JNJ_ret_1d", "EOG_EOGResources_vol_20d", "HangSeng_HK_ret_5d", "DIS_vol_20d", "PFE_ret_1d", "TED_Spread_vol_20d", "HUM_Humana_ret_5d", "US1Y_Rate_ret_5d", "Core_PCE_zscore_60d", "US3M_Rate_vol_20d", "ENB_EnbridgeInc_ret_1d", "SO_SouthernCo_ret_5d", "MO_AltriaMG_ret_1d", "hmm_p_stress", "LMT_LockheedMartin_vol_20d", "HD_ret_20d", "IYR_US_REIT2_zscore_60d"], "is_new": true}, {"model_id": "new_h10_GLOBAL_LogisticRegression_N20_t6", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 10, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "US3M_Rate_vol_20d", "DHR_ret_1d", "SBUX_ret_5d", "PAYX_Paychex_ret_20d", "VOD_Vodafone_zscore_60d", "XLK_Tech_zscore_60d", "LOW_Lowes_ret_5d", "HD_ret_5d", "ASX_Australia_ret_5d", "DHR_vol_20d", "MO_AltriaMG_ret_1d", "US3M_Rate_zscore_60d", "FedFunds_zscore_60d", "LMT_LockheedMartin_ret_1d", "LOW_Lowes_ret_20d", "PLD_Prologis_ret_5d", "EMR_Emerson_ret_20d", "EWY_Korea_ret_20d"], "is_new": true}, {"model_id": "new_h10_GLOBAL_LogisticRegression_N20_t7", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 10, "n_features": 20, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "spx_abs_ret_max_5d", "US5Y_Rate_ret_5d", "PLD_Prologis_ret_5d", "Industrial_Production_zscore_60d", "EWH_HongKong_ret_5d", "EFFR_ret_1d", "HangSeng_HK_vol_20d", "ENB_EnbridgeInc_ret_1d", "HUM_Humana_ret_5d", "XOM_ret_20d", "US3M_Rate_zscore_60d", "DHR_vol_20d", "3M_ret_5d", "heston_var_ev_h3", "XLB_Materials_zscore_60d", "NVDA_vol_20d", "EMR_Emerson_ret_20d", "IWM_SmallCap_vol_20d"], "is_new": true}, {"model_id": "new_h10_GLOBAL_LogisticRegression_N25_t0", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 10, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EMR_Emerson_ret_20d", "EWY_Korea_zscore_60d", "CCI_CrownCastle_vol_20d", "EWH_HongKong_ret_5d", "spx_abs_ret_max_5d", "heston_ev_h3", "CLX_Clorox_vol_20d", "LUV_SouthwestAir_ret_5d", "US7Y_Rate_ret_20d", "US1Y_Rate_ret_5d", "M_Macys_vol_20d", "MSTR_Bitcoin3_ret_1d", "NOC_Northrop_ret_20d", "ORCL_zscore_60d", "vix_acceleration_1d", "ORCL_vol_20d", "BTI_BritishAmerican_ret_5d", "US3M_Rate_zscore_60d", "NVDA_vol_20d", "MS_MorganStanley_zscore_60d", "EWC_Canada_zscore_60d", "SJM_JM_Smucker_ret_1d", "EWM_Malaysia_ret_1d"], "is_new": true}, {"model_id": "new_h10_GLOBAL_LogisticRegression_N25_t1", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 10, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "SJM_JM_Smucker_ret_5d", "HangSeng_HK_vol_20d", "DE_Deere_ret_5d", "3M_vol_20d", "EQIX_Equinix_ret_5d", "EWA_Australia_ret_1d", "EXC_Exelon_ret_1d", "DAX_Germany_zscore_60d", "IBEX_Spain_ret_20d", "heston_var_ev_h3", "DHR_ret_1d", "EMR_Emerson_ret_20d", "US5Y_Rate_ret_5d", "XLB_Materials_zscore_60d", "CPB_CampbellSoup_ret_5d", "PCAR_PaccarInc_ret_5d", "heston_var_ev_h5", "HD_zscore_60d", "HD_ret_5d", "AMZN_ret_5d", "BTI_BritishAmerican_ret_5d", "GD_GeneralDynamics_zscore_60d", "SJM_JM_Smucker_ret_1d"], "is_new": true}, {"model_id": "new_h10_GLOBAL_LogisticRegression_N25_t2", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 10, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "HangSeng_HK_ret_1d", "US5Y_Rate_ret_5d", "EWL_Switzerland_zscore_60d", "gjr_condvar_h1", "AMD_ret_5d", "IBEX_Spain_ret_20d", "INTC_ret_5d", "VVIX_ret_20d", "T_ret_1d", "CPB_CampbellSoup_ret_5d", "TM_Telephone_ret_1d", "CPB_CampbellSoup_ret_20d", "MO_AltriaMG_ret_1d", "BLK_BlackRock_zscore_60d", "HUM_Humana_ret_5d", "VRP_ma5", "QQQ_vol_20d", "TGT_Target_zscore_60d", "XOM_ret_20d", "EWS_Singapore_ret_5d", "GD_GeneralDynamics_zscore_60d", "MSTR_Bitcoin3_ret_5d", "Brent_Oil_FRED_ret_20d"], "is_new": true}, {"model_id": "new_h10_GLOBAL_LogisticRegression_N25_t3", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 10, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "HD_ret_20d", "T10Y2Y_Spread_ret_5d", "ORCL_zscore_60d", "EOG_EOGResources_vol_20d", "EWL_Switzerland_zscore_60d", "XOM_ret_1d", "US30Y_Rate_ret_20d", "INTC_ret_1d", "PG_ret_20d", "heston_ev_h3", "WTI_Oil_FRED_zscore_60d", "HD_ret_1d", "EOG_EOGResources_ret_5d", "AORD_AUS_zscore_60d", "CMCSA_ret_1d", "NEE_NextEra_ret_20d", "EWQ_France_zscore_60d", "Core_CPI_zscore_60d", "LOW_Lowes_ret_5d", "VOD_Vodafone_zscore_60d", "US5Y_Rate_ret_5d", "CPB_CampbellSoup_ret_5d", "ASX_Australia_vol_20d"], "is_new": true}, {"model_id": "new_h10_GLOBAL_LogisticRegression_N25_t4", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 10, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "TM_Telephone_ret_1d", "CLX_Clorox_vol_20d", "MS_MorganStanley_zscore_60d", "PG_ret_20d", "MO_AltriaMG_ret_1d", "EMR_Emerson_ret_20d", "TXN_vol_20d", "XLY_Disc_vol_20d", "SCHW_Schwab_ret_5d", "Retail_Sales_zscore_60d", "PPL_PPL_ret_1d", "SLB_Schlumberger_ret_5d", "heston_var_ev_h5", "HD_ret_5d", "LMT_LockheedMartin_vol_20d", "CPB_CampbellSoup_ret_20d", "AMD_ret_5d", "BLK_BlackRock_zscore_60d", "PCAR_PaccarInc_ret_5d", "HD_zscore_60d", "PAYX_Paychex_vol_20d", "XLK_Tech_zscore_60d", "MS_MorganStanley_ret_1d"], "is_new": true}, {"model_id": "new_h10_GLOBAL_LogisticRegression_N25_t5", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 10, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "HD_ret_20d", "BTI_BritishAmerican_ret_20d", "HD_ret_5d", "DAX_Germany_zscore_60d", "HangSeng_HK_vol_20d", "PFE_ret_1d", "US3Y_Rate_ret_5d", "US3M_Rate_vol_20d", "EWL_Switzerland_vol_20d", "XLB_Materials_zscore_60d", "LOW_Lowes_ret_20d", "NOC_Northrop_ret_20d", "DAX_Germany_vol_20d", "US5Y_Rate_ret_5d", "MS_MorganStanley_zscore_60d", "QQQ_vol_20d", "SBUX_vol_20d", "IYM_BasicMaterials_ret_20d", "LMT_LockheedMartin_vol_20d", "EWY_Korea_zscore_60d", "VOD_Vodafone_zscore_60d", "heston_ev_h3", "TXN_vol_20d"], "is_new": true}, {"model_id": "new_h10_GLOBAL_LogisticRegression_N25_t6", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 10, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "GD_GeneralDynamics_zscore_60d", "EWL_Switzerland_zscore_60d", "EWM_Malaysia_zscore_60d", "CPB_CampbellSoup_ret_20d", "TGT_Target_zscore_60d", "PG_ret_20d", "Industrial_Production_zscore_60d", "EWG_Germany_ret_20d", "IYM_BasicMaterials_ret_20d", "PPL_PPL_ret_1d", "AMD_ret_1d", "ENB_EnbridgeInc_ret_1d", "EFFR_vol_20d", "NFCI_ret_5d", "ORCL_zscore_60d", "SJM_JM_Smucker_ret_5d", "ES_Evergy_ret_1d", "HD_ret_20d", "EXC_Exelon_zscore_60d", "EWA_Australia_ret_1d", "HangSeng_HK_ret_1d", "PAYX_Paychex_vol_20d", "AMGN_Amgen_ret_1d"], "is_new": true}, {"model_id": "new_h10_GLOBAL_LogisticRegression_N25_t7", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 10, "n_features": 25, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "QQQ_vol_20d", "CTAS_Cintas_vol_20d", "ORCL_vol_20d", "XLK_Tech_zscore_60d", "JNJ_ret_1d", "Nikkei_Japan_zscore_60d", "US1Y_Rate_ret_20d", "NVDA_vol_20d", "BA_ret_1d", "CPB_CampbellSoup_ret_20d", "SCHW_Schwab_ret_5d", "PG_ret_20d", "TXN_vol_20d", "spx_abs_ret_max_5d", "3M_ret_5d", "SBUX_vol_20d", "SJM_JM_Smucker_ret_1d", "EXC_Exelon_zscore_60d", "DE_Deere_ret_5d", "DAX_Germany_vol_20d", "IYM_BasicMaterials_ret_20d", "hmm_p_stress", "PAYX_Paychex_ret_20d"], "is_new": true}, {"model_id": "new_h10_GLOBAL_LogisticRegression_N30_t0", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 10, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "AMD_ret_5d", "TM_Telephone_ret_1d", "ENB_EnbridgeInc_ret_1d", "EWG_Germany_vol_20d", "AMZN_ret_5d", "BA_ret_1d", "PPL_PPL_ret_1d", "HangSeng_HK_ret_5d", "MSTR_Bitcoin3_ret_5d", "DAX_Germany_zscore_60d", "XLB_Materials_zscore_60d", "EOG_EOGResources_ret_5d", "MSTR_Bitcoin3_ret_20d", "FedFunds_zscore_60d", "AMD_ret_1d", "MS_MorganStanley_zscore_60d", "CPB_CampbellSoup_zscore_60d", "HangSeng_HK_vol_20d", "DE_Deere_ret_5d", "EXC_Exelon_zscore_60d", "DOW_Price_zscore_60d", "Core_CPI_zscore_60d", "VVIX_ret_20d", "Michigan_Sentiment_ret_20d", "NFCI_ret_5d", "LMT_LockheedMartin_vol_20d", "CPB_CampbellSoup_ret_5d", "TXN_vol_20d"], "is_new": true}, {"model_id": "new_h10_GLOBAL_LogisticRegression_N30_t1", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 10, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "EOG_EOGResources_vol_20d", "NWL_Newell_ret_20d", "Brent_Oil_FRED_ret_20d", "LOW_Lowes_ret_20d", "heston_var_ev_h3", "PCAR_PaccarInc_ret_5d", "vix_acceleration_1d", "VOD_Vodafone_zscore_60d", "Michigan_Sentiment_ret_20d", "AMZN_ret_5d", "HD_ret_5d", "VRP_ma5", "spx_momentum_3d", "DAX_Germany_zscore_60d", "EFFR_ret_1d", "JNJ_ret_1d", "EXC_Exelon_zscore_60d", "CPB_CampbellSoup_vol_20d", "PG_ret_20d", "AORD_AUS_zscore_60d", "heston_var_ev_h5", "HUM_Humana_ret_5d", "AMD_ret_5d", "US30Y_Rate_ret_20d", "XLF_Fin_vol_20d", "EWC_Canada_zscore_60d", "DHR_vol_20d", "EWS_Singapore_ret_5d"], "is_new": true}, {"model_id": "new_h10_GLOBAL_LogisticRegression_N30_t2", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 10, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "XOM_ret_20d", "EWY_Korea_ret_20d", "Brent_Oil_FRED_ret_20d", "XLF_Fin_vol_20d", "JNJ_ret_1d", "EWC_Canada_zscore_60d", "CPB_CampbellSoup_zscore_60d", "XLY_Disc_vol_20d", "EOG_EOGResources_vol_20d", "CPB_CampbellSoup_ret_5d", "ASX_Australia_vol_20d", "NVDA_vol_20d", "US30Y_Rate_ret_20d", "IBEX_Spain_ret_20d", "ES_Evergy_ret_1d", "Nikkei_Japan_vol_20d", "Industrial_Production_zscore_60d", "SJM_JM_Smucker_ret_5d", "Brent_Oil_FRED_ret_5d", "PAYX_Paychex_zscore_60d", "Core_PCE_zscore_60d", "Retail_Sales_zscore_60d", "VVIX_ret_20d", "TM_Telephone_vol_20d", "AVB_AvalonBay_zscore_60d", "LUV_SouthwestAir_ret_5d", "EWJ_Japan_vol_20d", "AXP_Amex_ret_20d"], "is_new": true}, {"model_id": "new_h10_GLOBAL_LogisticRegression_N30_t3", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 10, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTE", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "GD_GeneralDynamics_zscore_60d", "CPB_CampbellSoup_vol_20d", "CPB_CampbellSoup_zscore_60d", "US3M_Rate_zscore_60d", "heston_ev_h3", "IWM_SmallCap_vol_20d", "ORCL_vol_20d", "XLF_Fin_vol_20d", "JNJ_ret_1d", "TED_Spread_zscore_60d", "AMZN_ret_5d", "M_Macys_vol_20d", "ITT_ITTInc_ret_5d", "DOW_Price_zscore_60d", "NOC_Northrop_ret_20d", "PAYX_Paychex_vol_20d", "GE_ret_1d", "spx_momentum_3d", "SBUX_zscore_60d", "MSTR_Bitcoin3_ret_20d", "AXP_Amex_vol_20d", "EWG_Germany_vol_20d", "HangSeng_HK_vol_20d", "Core_PCE_zscore_60d", "MSTR_Bitcoin3_ret_1d", "Nikkei_Japan_zscore_60d", "PAYX_Paychex_zscore_60d", "DE_Deere_ret_5d"], "is_new": true}, {"model_id": "new_h10_GLOBAL_LogisticRegression_N30_t4", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 10, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "Retail_Sales_zscore_60d", "NEE_NextEra_ret_20d", "ASX_Australia_ret_5d", "Core_PCE_zscore_60d", "PAYX_Paychex_ret_20d", "DAX_Germany_zscore_60d", "HD_ret_1d", "CI_Cigna_vol_20d", "AVB_AvalonBay_zscore_60d", "ENB_EnbridgeInc_ret_1d", "SLB_Schlumberger_ret_5d", "LLY_zscore_60d", "NFCI_ret_5d", "EWG_Germany_vol_20d", "AXP_Amex_vol_20d", "ORCL_zscore_60d", "CPB_CampbellSoup_ret_5d", "EWM_Malaysia_zscore_60d", "EXC_Exelon_ret_1d", "vix_acceleration_1d", "TM_Telephone_ret_1d", "EQIX_Equinix_ret_5d", "IYR_US_REIT2_zscore_60d", "SPY_zscore_60d", "BDX_Becton_Dickinson_ret_20d", "Core_CPI_zscore_60d", "IBEX_Spain_ret_20d", "Brent_Oil_FRED_ret_5d"], "is_new": true}, {"model_id": "new_h10_GLOBAL_LogisticRegression_N30_t5", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 10, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "EOG_EOGResources_vol_20d", "AVB_AvalonBay_zscore_60d", "MRK_Merck_zscore_60d", "gjr_condvar_h1", "VRP_ma5", "EWA_Australia_zscore_60d", "SPY_zscore_60d", "AMZN_ret_5d", "DE_Deere_ret_5d", "ASX_Australia_ret_5d", "Retail_Sales_zscore_60d", "MSTR_Bitcoin3_ret_5d", "CMCSA_ret_1d", "INTC_ret_1d", "MSTR_Bitcoin3_ret_1d", "CPB_CampbellSoup_ret_5d", "CI_Cigna_vol_20d", "QQQ_vol_20d", "MSTR_Bitcoin3_ret_20d", "LLY_zscore_60d", "EXC_Exelon_ret_1d", "Nikkei_Japan_zscore_60d", "EWH_HongKong_ret_5d", "AMD_ret_5d", "GD_GeneralDynamics_zscore_60d", "HangSeng_HK_ret_1d", "SJM_JM_Smucker_ret_5d", "EWA_Australia_ret_1d"], "is_new": true}, {"model_id": "new_h10_GLOBAL_LogisticRegression_N30_t6", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 10, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "BorderlineSMOTE", "best_params": "{}", "features": ["vix_zscore_10d", "VRP", "BTI_BritishAmerican_ret_5d", "CPB_CampbellSoup_zscore_60d", "JNJ_ret_1d", "EXC_Exelon_zscore_60d", "T10Y2Y_Spread_ret_5d", "VOD_Vodafone_zscore_60d", "AMT_AmericanTower_ret_1d", "heston_var_ev_h3", "XOM_ret_20d", "TGT_Target_zscore_60d", "US1Y_Rate_ret_20d", "CPB_CampbellSoup_ret_5d", "IYR_US_REIT2_zscore_60d", "XOM_ret_1d", "PAYX_Paychex_vol_20d", "T_ret_1d", "BLK_BlackRock_zscore_60d", "XLF_Fin_vol_20d", "EWA_Australia_ret_1d", "EWA_Australia_zscore_60d", "HangSeng_HK_vol_20d", "DOW_Price_zscore_60d", "TED_Spread_vol_20d", "AORD_AUS_zscore_60d", "HangSeng_HK_ret_1d", "Nikkei_Japan_zscore_60d", "MRK_Merck_zscore_60d", "US1Y_Rate_ret_5d"], "is_new": true}, {"model_id": "new_h10_GLOBAL_LogisticRegression_N30_t7", "algo": "LogisticRegression", "regime": "GLOBAL", "horizon": 10, "n_features": 30, "F1_dir": 0.0, "F1_UP_FORT": 0.0, "F1_DOWN_FORT": 0.0, "train_start": "2000-01-01", "sampler": "SMOTETomek", "best_params": "{}", "features": ["VRP", "vix_zscore_10d", "BLK_BlackRock_zscore_60d", "INTC_ret_1d", "EOG_EOGResources_ret_5d", "ES_Evergy_ret_1d", "DIS_vol_20d", "HD_zscore_60d", "MS_MorganStanley_zscore_60d", "EQR_Equity_ret_1d", "TGT_Target_zscore_60d", "SPY_zscore_60d", "NVDA_vol_20d", "DHR_ret_1d", "IWM_SmallCap_vol_20d", "EQIX_Equinix_ret_5d", "heston_var_ev_h7", "QQQ_vol_20d", "LLY_zscore_60d", "EWG_Germany_vol_20d", "IBEX_Spain_ret_20d", "BDX_Becton_Dickinson_ret_20d", "SJM_JM_Smucker_ret_5d", "MS_MorganStanley_ret_5d", "ASX_Australia_ret_5d", "Brent_Oil_FRED_ret_20d", "EWY_Korea_ret_20d", "SBUX_vol_20d", "HangSeng_HK_ret_1d", "ENB_EnbridgeInc_ret_1d"], "is_new": true}]')
df_runs=pd.DataFrame(ALL_RUNS)
existing=df_runs[~df_runs.get('is_new',pd.Series(False,index=df_runs.index)).fillna(False)]
new_runs=df_runs[df_runs.get('is_new',pd.Series(False,index=df_runs.index)).fillna(False)]
print(f"Panel total: {len(df_runs)} modèles")
print(f"  Existants (validés): {len(existing)}")
print(f"  Nouvelles combinaisons: {len(new_runs)}")
print(df_runs.groupby(['horizon','regime'])['model_id'].count().unstack(fill_value=0).to_string())


Panel total: 9465 modèles
  Existants (validés): 1785
  Nouvelles combinaisons: 7680
regime   CALM  GLOBAL  NORMAL  STRESS
horizon                              
1         449     406     420     448
2         320     320     320     320
3         478     416     472     401
5         428     437     434     395
7         415     483     450     373
10        320     320     320     320


In [5]:
def load_data(start=CONFIG['start_date']):
    t0=time.time()
    raw=yf.download(YF_TICKERS,start=start,auto_adjust=True,progress=False)['Close']
    raw.columns=[c.replace('^','IDX_').replace('-','_') for c in raw.columns]
    raw=raw.loc[:,raw.notna().mean()>=0.85].ffill().dropna(how='all')
    print(f"  YF: {raw.shape[0]}j × {raw.shape[1]} ({time.time()-t0:.1f}s)")
    fred_list=[]
    for name,sid in FRED_SERIES.items():
        try:
            s=web.DataReader(sid,'fred',start).squeeze(); s.name=f'FRED_{name}'
            fred_list.append(s)
        except Exception as e: print(f"  [WARN] {sid}: {e}")
    if fred_list:
        raw=pd.concat([raw,pd.concat(fred_list,axis=1).reindex(raw.index,method='ffill')],axis=1)
    print(f"  Total: {raw.shape} ({time.time()-t0:.1f}s)")
    return raw

df_raw=load_data()
all_dates=df_raw.dropna(how='all').index.sort_values()
SPLIT_IDX=int(len(all_dates)*0.80)
TEST_DATE=all_dates[SPLIT_IDX].strftime('%Y-%m-%d')
VIX_COL=[c for c in df_raw.columns if ('IDX_VIX' in c or c.endswith('_VIX')) and 'VXN' not in c and 'VVIX' not in c][0]
SPX_COLS=[c for c in df_raw.columns if 'GSPC' in c or c=='SPY']
SPX_COL=SPX_COLS[0] if SPX_COLS else None
print(f"Split: train→{all_dates[SPLIT_IDX-1].date()} | test→{all_dates[SPLIT_IDX].date()}")
print(f"VIX={VIX_COL} | SPX={SPX_COL}")


ERROR:yfinance:HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: AORD"}}}
ERROR:yfinance:
1 Failed download:
ERROR:yfinance:['AORD']: YFTzMissingError('possibly delisted; no timezone found')


  YF: 6907j × 56 (37.8s)
  Total: (6907, 60) (40.3s)
Split: train→2021-03-29 | test→2021-03-30
VIX=IDX_VIX | SPX=SPY


In [6]:
def build_features(df_raw,vix_col,spx_col,split_idx):
    t0=time.time(); feats={}
    vix=df_raw[vix_col].replace([np.inf,-np.inf],np.nan).ffill().bfill()
    vix_ret=np.log(vix/vix.shift(1)).replace([np.inf,-np.inf],np.nan).fillna(0)
    spx_ret=pd.Series(0.,index=df_raw.index)
    if spx_col:
        spx=df_raw[spx_col].ffill().bfill()
        spx_ret=np.log(spx/spx.shift(1)).fillna(0)

    # Rendements multi-horizons pour tous les tickers
    print(f"  [FEAT] Rendements ({df_raw.shape[1]} séries)...")
    for col in df_raw.columns:
        s=df_raw[col].replace([np.inf,-np.inf],np.nan).ffill().bfill()
        lr=np.log(s/s.shift(1)).replace([np.inf,-np.inf],np.nan)
        for w in [1,5,20]:
            feats[f'{col}_ret_{w}d']=np.log(s/s.shift(w)).replace([np.inf,-np.inf],np.nan)
        feats[f'{col}_vol_20d']=lr.rolling(20,min_periods=10).std()
        mu=s.rolling(60,min_periods=30).mean(); sd=s.rolling(60,min_periods=30).std().replace(0,np.nan)
        feats[f'{col}_zscore_60d']=(s-mu)/sd

    # VIX features
    for w in [5,10,20]:
        ma=vix.rolling(w,min_periods=w//2).mean(); sd=vix.rolling(w,min_periods=w//2).std().replace(0,np.nan)
        feats[f'vix_zscore_{w}d']=(vix-ma)/sd
        feats[f'vix_vs_ma{w}']=(vix-ma)/ma.replace(0,np.nan)
    feats['vix_level']=vix; feats['vix_ma_20']=vix.rolling(20,min_periods=10).mean()
    for w in [5,10]: feats[f'vix_vol_of_vol_{w}d']=vix_ret.rolling(w,min_periods=w//2).std()
    for w in [2,3,5]: feats[f'vix_momentum_{w}d']=vix.pct_change(w)
    feats['vix_acceleration_1d']=vix_ret-vix_ret.shift(1)
    feats['vix_acceleration_3d']=vix_ret-vix_ret.shift(3)
    feats['vix_erratic_ratio']=vix_ret.abs().rolling(5,min_periods=3).max()/vix_ret.abs().rolling(5,min_periods=3).mean().replace(0,np.nan)
    feats['vix_vol_ratio_5_60']=vix_ret.rolling(5,min_periods=3).std()/vix_ret.rolling(60,min_periods=30).std().replace(0,np.nan)
    feats['vix_max_abs_ret_5d']=vix_ret.abs().rolling(5,min_periods=3).max()
    feats['vix_mean_abs_ret_5d']=vix_ret.abs().rolling(5,min_periods=3).mean()
    if spx_col:
        spx=df_raw[spx_col].ffill().bfill()
        feats['spx_drawdown_252d']=(spx-spx.rolling(252,min_periods=126).max())/spx.rolling(252,min_periods=126).max().replace(0,np.nan)
        feats['spx_vol_5d']=spx_ret.rolling(5,min_periods=3).std()
        feats['spx_abs_ret_max_5d']=spx_ret.abs().rolling(5,min_periods=3).max()
        feats['spx_momentum_3d']=spx.pct_change(3)
        feats['vix_spx_corr_30d']=vix_ret.rolling(30,min_periods=15).corr(spx_ret)

    # EGARCH
    print(f"  [FEAT] EGARCH ({time.time()-t0:.0f}s)...")
    try:
        sp_tr=(spx_ret.iloc[:split_idx]*100)
        am=arch_model(sp_tr,vol='EGARCH',p=1,q=1,dist='skewt',rescale=False)
        res=am.fit(disp='off',show_warning=False)
        fc=res.forecast(start=0,reindex=True)
        cv=(fc.variance.iloc[:,0]/10000).reindex(df_raw.index,method='ffill').replace([np.inf,-np.inf],np.nan)
        feats['egarch_condvar']=cv; feats['egarch_delta']=cv.diff()
        for h in [1,3,5]:
            feats[f'EGARCH_SPX_condvar_h{h}']=cv; feats[f'EGARCH_SPX_delta_h{h}']=cv.diff()
        # GJR-GARCH
        am2=arch_model(sp_tr,vol='GARCH',p=1,o=1,q=1,dist='skewt',rescale=False)
        res2=am2.fit(disp='off',show_warning=False)
        fc2=res2.forecast(start=0,reindex=True)
        cv2=(fc2.variance.iloc[:,0]/10000).reindex(df_raw.index,method='ffill').replace([np.inf,-np.inf],np.nan)
        feats['gjr_condvar']=cv2; feats['gjr_delta']=cv2.diff()
        print(f"    EGARCH+GJR OK")
    except Exception as e: print(f"    [WARN] GARCH: {e}")

    # Kalman (+1j shift anti-leakage)
    print(f"  [FEAT] Kalman ({time.time()-t0:.0f}s)...")
    try:
        vc=vix.interpolate('linear').ffill().bfill().astype(float)
        kf=KalmanFilter(transition_matrices=[[1.]],observation_matrices=[[1.]],
                        initial_state_mean=[float(vc.iloc[0])],
                        initial_state_covariance=[[1.]],
                        em_vars=['transition_covariance','observation_covariance'])
        kf=kf.em(vc.iloc[:split_idx].values.reshape(-1,1),n_iter=20)
        smf,_=kf.filter(vc.values.reshape(-1,1)); sms,_=kf.smooth(vc.values.reshape(-1,1))
        kf_f=pd.Series(smf[:,0],index=df_raw.index); kf_s=pd.Series(sms[:,0],index=df_raw.index)
        feats['VIX_Residual']=(vc-kf_f).shift(1).replace([np.inf,-np.inf],np.nan)
        feats['VIX_Innovation']=(vc-kf_s.shift(1)).shift(1).replace([np.inf,-np.inf],np.nan)
        feats['kalman_residual']=feats['VIX_Residual']
        feats['kalman_innovation']=feats['VIX_Innovation']
        feats['kalman_filtered']=kf_f
        print(f"    Kalman OK (Q={kf.transition_covariance[0,0]:.5f})")
    except Exception as e: print(f"    [WARN] Kalman: {e}")

    # HMM K=2
    print(f"  [FEAT] HMM ({time.time()-t0:.0f}s)...")
    try:
        rv5=vix_ret.pow(2).rolling(5,min_periods=3).mean()
        mu_t=vix.iloc[:split_idx].mean(); sd_t=vix.iloc[:split_idx].std()
        vix_n=(vix-mu_t)/(sd_t if sd_t>1e-8 else 1)
        X_hmm=pd.DataFrame({'r':vix_ret,'v':np.sqrt(rv5.clip(0)),'l':vix_n}).dropna()
        mh=hmmlib.GaussianHMM(n_components=2,covariance_type='full',n_iter=200,random_state=SEED)
        mh.fit(X_hmm.iloc[:split_idx].values)
        st=mh.predict(X_hmm.iloc[:split_idx].values)
        rv_v=[rv5.reindex(X_hmm.index[:split_idx]).values[st==s].mean() if (st==s).any() else 0 for s in range(2)]
        ss=int(np.argmax(rv_v))
        pr=mh.predict_proba(X_hmm.values)
        p_stress=pd.Series(pr[:,ss],index=X_hmm.index).reindex(df_raw.index)
        feats['P_stress_HMM']=p_stress; feats['hmm_p_stress']=p_stress
        feats['hmm_state']=pd.Series(mh.predict(X_hmm.values),index=X_hmm.index).reindex(df_raw.index)
        print(f"    HMM OK (état stress={ss})")
    except Exception as e: print(f"    [WARN] HMM: {e}")

    # Heston proxies + espérances conditionnelles
    print(f"  [FEAT] Heston ({time.time()-t0:.0f}s)...")
    try:
        v0=(vix/100).pow(2); theta=vix_ret.pow(2).rolling(60,min_periods=30).mean()
        vvix_c=[c for c in df_raw.columns if 'VVIX' in c]
        xi=(df_raw[vvix_c[0]].ffill().bfill()/100) if vvix_c else vix_ret.rolling(20).std()
        xi=xi.reindex(df_raw.index,method='ffill').replace([np.inf,-np.inf],np.nan)
        rho=vix_ret.rolling(30,min_periods=15).corr(spx_ret)
        rho_60=vix_ret.rolling(60,min_periods=30).corr(spx_ret)
        # Kappa via half-life AR(1) rolling 252j
        def rolling_kappa(s,w=252):
            k=pd.Series(np.nan,index=s.index); sf=s.ffill().bfill()
            for i in range(w,len(sf)):
                try:
                    b=np.corrcoef(sf.iloc[i-w:i].values,sf.iloc[i-w+1:i+1].values)[0,1]
                    if np.isfinite(b) and 0<abs(b)<0.9999:
                        hl=-np.log(2)/np.log(abs(b))
                        if np.isfinite(hl) and hl>0: k.iloc[i]=np.log(2)/hl
                except: pass
            return k.replace([np.inf,-np.inf],np.nan)
        kappa=rolling_kappa(vix)
        feats['heston_v0']=v0; feats['heston_theta']=theta
        feats['heston_xi']=xi; feats['heston_rho']=rho; feats['heston_rho_60']=rho_60
        feats['heston_kappa']=kappa
        feats['heston_feller']=(2*kappa*theta)/xi.pow(2).replace(0,np.nan)
        feats['heston_v0_minus_theta']=v0-theta
        mu_xi=xi.iloc[:split_idx].mean(); sd_xi=xi.iloc[:split_idx].std()
        feats['heston_xi_zscore']=(xi-mu_xi)/(sd_xi if sd_xi>1e-8 else 1)
        feats['heston_xi_ma5']=xi.rolling(5,min_periods=3).mean()
        mu_k=kappa.iloc[:split_idx].mean(); sd_k=kappa.iloc[:split_idx].std()
        feats['heston_kappa_zscore']=(kappa-mu_k)/(sd_k if sd_k>1e-8 else 1)
        for h in [1,3,5,7,10]:
            ev=(theta+(v0-theta)*np.exp(-kappa*h)).replace([np.inf,-np.inf],np.nan)
            var_ev=(v0*xi**2*np.exp(-kappa*h)*(1-np.exp(-kappa*h))/kappa.replace(0,np.nan)
                    +theta*xi**2*(1-np.exp(-kappa*h))**2/(2*kappa.replace(0,np.nan))).replace([np.inf,-np.inf],np.nan)
            feats[f'heston_ev_h{h}']=ev
            feats[f'heston_spread_h{h}']=v0-ev
            feats[f'heston_vol_h{h}']=np.sqrt(ev.clip(lower=0))*100
            feats[f'heston_var_ev_h{h}']=var_ev
        print(f"    Heston OK (xi_mean={xi.iloc[:split_idx].mean():.4f})")
    except Exception as e: print(f"    [WARN] Heston: {e}")

    # VRP
    print(f"  [FEAT] VRP ({time.time()-t0:.0f}s)...")
    try:
        rv1=vix_ret.pow(2).replace([np.inf,-np.inf],np.nan)
        rv5d=rv1.rolling(5,min_periods=3).mean(); rv22d=rv1.rolling(22,min_periods=10).mean()
        rv_tgt=rv1.shift(-22).rolling(22,min_periods=11).mean()
        hdf=pd.DataFrame({'rv1':rv1,'rv5':rv5d,'rv22':rv22d,'y':rv_tgt}).dropna().replace([np.inf,-np.inf],np.nan).dropna()
        htr=hdf.iloc[:split_idx]
        Xh=sm.add_constant(htr[['rv1','rv5','rv22']],has_constant='add')
        hm=sm.OLS(htr['y'],Xh).fit()
        Xf=sm.add_constant(hdf[['rv1','rv5','rv22']],has_constant='add').fillna(0)
        rv_pred=hm.predict(Xf).reindex(df_raw.index).fillna(0)
        vrp=(vix/100).pow(2)-rv_pred
        mu_v=vrp.iloc[:split_idx].mean(); sd_v=vrp.iloc[:split_idx].std()
        feats['VRP']=vrp; feats['VRP_zscore']=(vrp-mu_v)/(sd_v if sd_v>1e-8 else 1)
        feats['VRP_ma5']=vrp.rolling(5,min_periods=3).mean()
        print(f"    VRP OK (R²={hm.rsquared:.4f})")
    except Exception as e: print(f"    [WARN] VRP: {e}")

    # Jump + Hawkes
    print(f"  [FEAT] Jump+Hawkes ({time.time()-t0:.0f}s)...")
    sig60=vix_ret.rolling(60,min_periods=30).std()
    is_j=(vix_ret.abs()>3*sig60).astype(float)
    feats['jump_intensity_20d']=is_j.rolling(20,min_periods=10).mean()
    feats['jump_intensity_60d']=is_j.rolling(60,min_periods=30).mean()
    try:
        sig_hw=vix_ret.rolling(30,min_periods=15).std()
        jt=vix_ret.index[vix_ret.abs()>2*sig_hw]
        hw=pd.Series(0.,index=vix_ret.index)
        for i,t in enumerate(vix_ret.index):
            past=jt[jt<t]
            hw.iloc[i]=0.3+0.3*float(np.sum(np.exp(-0.1*np.array([(t-tj).days for tj in past],dtype=float)))) if len(past) else 0.3
        mu_hw=hw.iloc[:split_idx].mean(); sd_hw=hw.iloc[:split_idx].std()
        feats['hawkes_intensity']=hw; feats['hawkes_zscore']=(hw-mu_hw)/(sd_hw if sd_hw>1e-8 else 1)
    except Exception as e: print(f"    [WARN] Hawkes: {e}")

    # Implied correlation proxy
    try:
        sec=[c for c in df_raw.columns if any(s in c for s in ['XLK','XLF','XLE','XLV','XLU','XLB','XLI','XLY'])]
        if sec:
            vix_sq=(vix/100).pow(2); w=1./len(sec)
            sv_sum=sum(w**2*np.log(df_raw[c].ffill()/df_raw[c].ffill().shift(1)).rolling(21,min_periods=10).std().pow(2) for c in sec)
            feats['impl_corr_proxy']=(vix_sq-sv_sum).clip(-1,1)
    except: pass

    df_feat=pd.DataFrame(feats,index=df_raw.index).replace([np.inf,-np.inf],np.nan)
    df_full=pd.concat([df_raw,df_feat],axis=1)
    df_full=df_full.loc[:,~df_full.columns.duplicated()]
    print(f"  [FEAT] Total: {df_full.shape[1]} cols ({time.time()-t0:.0f}s)")
    return df_full

print("[FEATURES] Démarrage...")
df_features=build_features(df_raw,VIX_COL,SPX_COL,SPLIT_IDX)
print(f"Dataset: {df_features.shape}")


[FEATURES] Démarrage...
  [FEAT] Rendements (60 séries)...
  [FEAT] EGARCH (0s)...
    EGARCH+GJR OK
  [FEAT] Kalman (1s)...


    Kalman OK (Q=2.02354)
  [FEAT] HMM (55s)...
    HMM OK (état stress=0)
  [FEAT] Heston (55s)...
    Heston OK (xi_mean=0.0645)
  [FEAT] VRP (57s)...
    VRP OK (R²=0.0493)
  [FEAT] Jump+Hawkes (57s)...
  [FEAT] Total: 441 cols (63s)
Dataset: (6907, 441)


In [7]:
def build_target(vix_series,horizon,split_idx):
    vix=vix_series.ffill().bfill(); vix_tr=vix.iloc[:split_idx]
    calm_thr=vix_tr.quantile(0.33); stress_thr=vix_tr.quantile(0.67)
    regime=pd.Series('NORMAL',index=vix.index)
    regime[vix<calm_thr]='CALM'; regime[vix>=stress_thr]='STRESS'
    ret=(vix.shift(-horizon)/vix)-1
    flat=ret.abs()<CONFIG['flat_thr']
    ret=ret.loc[~flat].dropna(); reg_r=regime.reindex(ret.index)
    ret_tr=ret.iloc[:split_idx]; reg_tr=reg_r.iloc[:split_idx]
    thr={}
    for reg in ['CALM','NORMAL','STRESS']:
        sub=ret_tr[reg_tr==reg]
        thr[reg]=(sub.quantile(0.25) if len(sub)>=20 else ret_tr.quantile(0.25),
                  sub.quantile(0.75) if len(sub)>=20 else ret_tr.quantile(0.75))
    thr['GLOBAL']=(ret_tr.quantile(0.25),ret_tr.quantile(0.75))
    def classify(r,reg):
        q25,q75=thr.get(reg,(0,0))
        if r<q25: return 0
        if r<0:   return 1
        if r<q75: return 2
        return 3
    target=pd.Series([classify(r,reg_r[i]) for i,r in ret.items()],index=ret.index,name=TARGET_COL)
    return target,reg_r,thr

def recon_feat(fname,df,w=20,eps=1e-8):
    if fname in df.columns: return df[fname]
    for sep in ['__div__','__minus__','__prod__','__zrel__','__macross__','__ret5x__']:
        if sep in fname:
            a,b=fname.split(sep,1)
            if a not in df.columns or b not in df.columns: return None
            si,sj=df[a],df[b]
            if sep=='__div__':    return si/sj.where(sj.abs()>=eps,np.nan)
            if sep=='__minus__':  return si-sj
            if sep=='__prod__':   return si*sj
            if sep=='__zrel__':
                d=si-sj; rs=d.rolling(w,min_periods=w//2).std(); return d/rs.replace(0,np.nan)
            if sep=='__macross__':
                mi=si.rolling(w,min_periods=w//2).mean(); mj=sj.rolling(w,min_periods=w//2).mean()
                return mi/mj.where(mj.abs()>=eps,np.nan)
            if sep=='__ret5x__':  return si.pct_change(5)*sj
    return None

def build_X(df,feats):
    cols={};
    for f in feats:
        s=recon_feat(f,df)
        if s is not None: cols[f]=s
    return pd.DataFrame(cols,index=df.index).replace([np.inf,-np.inf],np.nan) if cols else pd.DataFrame(index=df.index)

def get_clf(algo,params_str='{}'):
    try: p=json.loads(params_str) if params_str not in ('{}','nan','None','') else {}
    except: p={}
    def g(k,d): return p.get(k,d)
    if 'XGBoost' in algo:
        return XGBClassifier(n_estimators=g('n_estimators',200),max_depth=g('max_depth',4),
            learning_rate=g('learning_rate',0.05),subsample=g('subsample',0.8),
            colsample_bytree=g('colsample_bytree',0.8),min_child_weight=g('min_child_weight',3),
            eval_metric='mlogloss',objective='multi:softprob',random_state=SEED,n_jobs=-1,verbosity=0)
    if 'LightGBM' in algo:
        return LGBMClassifier(n_estimators=g('n_estimators',200),max_depth=g('max_depth',5),
            learning_rate=g('learning_rate',0.05),num_leaves=g('num_leaves',31),
            min_child_samples=g('min_child_samples',10),subsample=g('subsample',0.8),
            class_weight='balanced',random_state=SEED,verbose=-1,n_jobs=-1)
    if 'GradientBoosting' in algo:
        return GradientBoostingClassifier(n_estimators=g('n_estimators',200),
            learning_rate=g('learning_rate',0.05),max_depth=g('max_depth',4),
            min_samples_leaf=g('min_samples_leaf',10),subsample=g('subsample',0.8),random_state=SEED)
    if 'RandomForest' in algo:
        return RandomForestClassifier(n_estimators=g('n_estimators',200),max_depth=g('max_depth',6),
            min_samples_leaf=g('min_samples_leaf',5),class_weight='balanced',random_state=SEED,n_jobs=-1)
    if 'ExtraTrees' in algo:
        return ExtraTreesClassifier(n_estimators=200,max_depth=6,class_weight='balanced',
                                     random_state=SEED,n_jobs=-1)
    if 'LogisticRegression' in algo:
        return LogisticRegression(C=g('C',1.0),max_iter=2000,class_weight='balanced',
            multi_class='multinomial',solver='lbfgs',random_state=SEED)
    raise ValueError(f"Algo inconnu: {algo}")

def get_samp(name):
    m={'SMOTE':SMOTE(random_state=SEED),
       'BorderlineSMOTE':BorderlineSMOTE(random_state=SEED,kind='borderline-1'),
       'SMOTETomek':SMOTETomek(random_state=SEED),
       'SMOTEENN':SMOTEENN(random_state=SEED),
       'ADASYN':ADASYN(random_state=SEED)}
    return m.get(name,SMOTE(random_state=SEED))

def align4(clf,X):
    p=clf.predict_proba(X)
    if p.shape[1]==4: return p
    fp=np.full((len(p),4),0.25)
    for ci,c in enumerate(clf.classes_): fp[:,int(c)]=p[:,ci]
    return fp

def metrics(y_true,y_pred):
    dm={0:'DOWN',1:'DOWN',2:'UP',3:'UP'}
    yd_t=[dm[y] for y in y_true]; yd_p=[dm[y] for y in y_pred]
    m={'F1_4cls':round(f1_score(y_true,y_pred,average='macro',zero_division=0),4),
       'Acc_4cls':round(accuracy_score(y_true,y_pred),4),
       'Acc_dir':round(accuracy_score(yd_t,yd_p),4),
       'F1_dir':round(f1_score(yd_t,yd_p,average='macro',zero_division=0),4),
       'F1_UP':round(f1_score(yd_t,yd_p,pos_label='UP',average='binary',zero_division=0),4),
       'F1_DOWN':round(f1_score(yd_t,yd_p,pos_label='DOWN',average='binary',zero_division=0),4)}
    ui=[i for i,y in enumerate(y_true) if dm[y]=='UP']
    di=[i for i,y in enumerate(y_true) if dm[y]=='DOWN']
    if len(ui)>=10:
        yt=['FORT' if y_true[i]==3 else 'FAIBLE' for i in ui]
        yp=['FORT' if y_pred[i]==3 else 'FAIBLE' for i in ui]
        m['F1_UP_FORT']=round(f1_score(yt,yp,pos_label='FORT',average='binary',zero_division=0),4)
    else: m['F1_UP_FORT']=0.
    if len(di)>=10:
        yt=['FORT' if y_true[i]==0 else 'FAIBLE' for i in di]
        yp=['FORT' if y_pred[i]==0 else 'FAIBLE' for i in di]
        m['F1_DOWN_FORT']=round(f1_score(yt,yp,pos_label='FORT',average='binary',zero_division=0),4)
    else: m['F1_DOWN_FORT']=0.
    return m

# Targets pour tous les horizons
print("Construction des cibles...")
TARGETS={}
for h in CONFIG['horizons']:
    tgt,reg_s,thr=build_target(df_features[VIX_COL],h,SPLIT_IDX)
    TARGETS[h]={'target':tgt,'regime':reg_s,'thresholds':thr}
    n_tr=(tgt.index<pd.Timestamp(TEST_DATE)).sum()
    n_te=(tgt.index>=pd.Timestamp(TEST_DATE)).sum()
    print(f"  h={h}j: {len(tgt)} obs (tr={n_tr} te={n_te}) {tgt.value_counts().to_dict()}")


Construction des cibles...
  h=1j: 6379 obs (tr=5099 te=1280) {1: 1862, 0: 1604, 3: 1596, 2: 1317}
  h=2j: 6659 obs (tr=5320 te=1339) {1: 1878, 3: 1672, 0: 1665, 2: 1444}
  h=3j: 6728 obs (tr=5387 te=1341) {1: 1911, 3: 1697, 0: 1677, 2: 1443}
  h=5j: 6767 obs (tr=5412 te=1355) {1: 1922, 3: 1708, 0: 1688, 2: 1449}
  h=7j: 6773 obs (tr=5421 te=1352) {1: 1894, 3: 1732, 0: 1690, 2: 1457}
  h=10j: 6775 obs (tr=5425 te=1350) {1: 1927, 3: 1732, 0: 1688, 2: 1428}


In [8]:
# ============================================================
# ARCHITECTURES DEEP LEARNING
# LSTM, TCN, Transformer, CNN-LSTM, TFT, N-BEATS, Mamba
# ============================================================
class PreScaledDS(torch.utils.data.Dataset):
    def __init__(self,X,y,lb=CONFIG['dl_lookback']):
        self.X=np.nan_to_num(X.astype(np.float32),nan=0.,posinf=0.,neginf=0.)
        self.y=y.astype(np.int64); self.lb=lb
    def __len__(self): return max(0,len(self.X)-self.lb)
    def __getitem__(self,i):
        return torch.tensor(self.X[i:i+self.lb]),torch.tensor(self.y[i+self.lb])

class FocalLoss(nn.Module):
    def __init__(self,gamma=2.,alpha=None,ls=0.1):
        super().__init__(); self.g=gamma; self.a=alpha; self.ls=ls
    def forward(self,logits,targets):
        n=logits.size(1)
        oh=torch.zeros_like(logits).scatter_(1,targets.unsqueeze(1),1)
        smooth=oh*(1-self.ls)+self.ls/n
        lp=torch.log_softmax(logits,dim=1); p=lp.exp()
        at=self.a.to(logits.device)[targets].unsqueeze(1) if self.a is not None else 1.
        return (-(at*(1-p)**self.g*smooth*lp).sum(dim=1)).mean()

class VIX_LSTM(nn.Module):
    def __init__(self,d,h=128,nl=2,drop=CONFIG['dl_dropout'],nc=4):
        super().__init__()
        self.lstm=nn.LSTM(d,h,nl,batch_first=True,dropout=drop)
        self.norm=nn.LayerNorm(h)
        self.fc=nn.Sequential(nn.Linear(h,64),nn.GELU(),nn.Dropout(drop),nn.Linear(64,nc))
    def forward(self,x): out,_=self.lstm(x); return self.fc(self.norm(out[:,-1,:]))

class TCNBlock(nn.Module):
    def __init__(self,ni,no,ks,dil,drop=0.2):
        super().__init__()
        pad=(ks-1)*dil
        self.conv=nn.utils.weight_norm(nn.Conv1d(ni,no,ks,padding=pad,dilation=dil))
        self.drop=nn.Dropout(drop); self.act=nn.GELU()
        self.skip=nn.Conv1d(ni,no,1) if ni!=no else None
    def forward(self,x):
        out=self.act(self.drop(self.conv(x)[:,:,:-self.conv.padding[0]] if self.conv.padding[0]>0 else self.conv(x)))
        return self.act(out+(x if self.skip is None else self.skip(x)))

class VIX_TCN(nn.Module):
    def __init__(self,d,ch=[32,64,128],ks=3,drop=CONFIG['dl_dropout'],nc=4):
        super().__init__()
        lys=[]; ic=d
        for i,c in enumerate(ch): lys.append(TCNBlock(ic,c,ks,2**i,drop)); ic=c
        self.net=nn.Sequential(*lys)
        self.fc=nn.Sequential(nn.Linear(ch[-1],64),nn.GELU(),nn.Dropout(drop),nn.Linear(64,nc))
    def forward(self,x): return self.fc(self.net(x.transpose(1,2)).mean(dim=2))

class VIX_Transformer(nn.Module):
    def __init__(self,d,dm=128,nh=4,nl=3,drop=CONFIG['dl_dropout'],nc=4):
        super().__init__()
        self.proj=nn.Linear(d,dm); self.pos=nn.Embedding(CONFIG['dl_lookback'],dm)
        enc=nn.TransformerEncoderLayer(dm,nh,dm*4,drop,batch_first=True,norm_first=True)
        self.enc=nn.TransformerEncoder(enc,nl)
        self.fc=nn.Sequential(nn.Linear(dm,64),nn.GELU(),nn.Dropout(drop),nn.Linear(64,nc))
    def forward(self,x):
        B,T,_=x.shape; pos=torch.arange(T,device=x.device).unsqueeze(0).expand(B,-1)
        return self.fc(self.enc(self.proj(x)+self.pos(pos))[:,-1,:])

class VIX_CNNLSTM(nn.Module):
    def __init__(self,d,cf=64,lh=128,ks=3,drop=CONFIG['dl_dropout'],nc=4):
        super().__init__()
        self.conv=nn.Sequential(nn.Conv1d(d,cf,ks,padding='same'),nn.GELU(),nn.BatchNorm1d(cf),nn.Dropout(drop))
        self.lstm=nn.LSTM(cf,lh,batch_first=True,num_layers=2,dropout=drop)
        self.fc=nn.Sequential(nn.Linear(lh,64),nn.GELU(),nn.Dropout(drop),nn.Linear(64,nc))
    def forward(self,x): x=self.conv(x.transpose(1,2)).transpose(1,2); out,_=self.lstm(x); return self.fc(out[:,-1,:])

class GLU(nn.Module):
    def __init__(self,d): super().__init__(); self.f=nn.Linear(d,d); self.g=nn.Linear(d,d)
    def forward(self,x): return self.f(x)*torch.sigmoid(self.g(x))

class VIX_TFT(nn.Module):
    def __init__(self,d,dm=128,nh=4,nl=2,drop=CONFIG['dl_dropout'],nc=4):
        super().__init__()
        self.vsn=nn.Sequential(nn.Linear(d,dm),nn.GELU(),nn.Linear(dm,d),nn.Softmax(dim=-1))
        self.proj=nn.Linear(d,dm)
        self.lstm=nn.LSTM(dm,dm,num_layers=nl,batch_first=True,dropout=drop)
        self.ln1=nn.LayerNorm(dm)
        enc=nn.TransformerEncoderLayer(dm,nh,dm*2,drop,batch_first=True,norm_first=True)
        self.attn=nn.TransformerEncoder(enc,2)
        self.glu=GLU(dm); self.ln2=nn.LayerNorm(dm)
        self.fc=nn.Sequential(nn.Linear(dm,64),nn.GELU(),nn.Dropout(drop),nn.Linear(64,nc))
    def forward(self,x):
        w=self.vsn(x.mean(dim=1,keepdim=True)); x=x*w; x=self.proj(x)
        lo,_=self.lstm(x); lo=self.ln1(lo+x)
        ao=self.attn(lo); out=self.ln2(self.glu(ao)+ao)
        return self.fc(out[:,-1,:])

class SSMLayer(nn.Module):
    def __init__(self,d,ks=21):
        super().__init__()
        self.k=nn.Parameter(torch.randn(d,1,ks)*0.01)
        self.n=nn.LayerNorm(d); self.ig=nn.Linear(d,d); self.og=nn.Linear(d,d)
    def forward(self,x):
        B,L,D=x.shape; xp=F.pad(x.transpose(1,2),(self.k.shape[2]-1,0))
        out=F.conv1d(xp,torch.softmax(self.k,dim=-1),groups=D).transpose(1,2)
        out=out*torch.sigmoid(self.ig(x)); out=self.n(out+x)
        return out*torch.sigmoid(self.og(out))

class VIX_Mamba(nn.Module):
    def __init__(self,d,dm=128,nl=4,drop=CONFIG['dl_dropout'],nc=4):
        super().__init__()
        self.proj=nn.Linear(d,dm)
        self.lys=nn.ModuleList([SSMLayer(dm) for _ in range(nl)])
        self.drop=nn.Dropout(drop); self.n=nn.LayerNorm(dm)
        self.fc=nn.Sequential(nn.Linear(dm,64),nn.GELU(),nn.Dropout(drop),nn.Linear(64,nc))
    def forward(self,x):
        x=self.proj(x)
        for l in self.lys: x=self.drop(l(x))
        return self.fc(self.n(x[:,-1,:]))

def compute_cw(y,nc=4):
    c=np.bincount(y,minlength=nc); w=1./(c+1e-6)
    return torch.tensor(w/w.sum()*nc,dtype=torch.float32)

def train_dl(model,dl_tr,dl_va,cw=None,epochs=CONFIG['dl_epochs'],lr=CONFIG['dl_lr'],label='',patience=8):
    crit=FocalLoss(gamma=2.,alpha=cw,ls=0.1)
    opt=torch.optim.AdamW(model.parameters(),lr=lr,weight_decay=1e-4)
    sched=torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(opt,T_0=10,T_mult=2)
    best_loss,best_st,wait=float('inf'),None,0; t0=time.time()
    for ep in range(epochs):
        model.train()
        for bx,by in dl_tr:
            bx,by=bx.to(device),by.to(device); opt.zero_grad()
            loss=crit(model(bx),by); loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(),1.); opt.step()
        sched.step()
        model.eval(); vl=0.
        with torch.no_grad():
            for bx,by in dl_va: bx,by=bx.to(device),by.to(device); vl+=crit(model(bx),by).item()
        if vl<best_loss: best_loss=vl; best_st={k:v.cpu().clone() for k,v in model.state_dict().items()}; wait=0
        else:
            wait+=1
            if wait>=patience: print(f"  [{label}] Stop ep{ep+1} ({time.time()-t0:.0f}s)"); break
        if (ep+1)%10==0: print(f"  [{label}] ep{ep+1} val_loss={vl:.4f} ({time.time()-t0:.0f}s)")
    if best_st: model.load_state_dict(best_st)

def eval_dl(model,loader):
    model.eval(); preds,probs,targets=[],[],[]
    with torch.no_grad():
        for bx,by in loader:
            logits=model(bx.to(device))
            preds.extend(logits.argmax(1).cpu().numpy())
            probs.extend(torch.softmax(logits,dim=1).cpu().numpy())
            targets.extend(by.numpy())
    return np.array(targets),np.array(preds),np.array(probs)

print("Architectures DL définies: LSTM, TCN, Transformer, CNN-LSTM, TFT, Mamba")


Architectures DL définies: LSTM, TCN, Transformer, CNN-LSTM, TFT, Mamba


In [ ]:
# ============================================================
# ENTRAÎNEMENT DU PANEL ML COMPLET (9465 modèles)
# Temps estimé : 4-8h sur Google Colab
# ============================================================
trained_clfs={}; trained_sc={}; trained_fc={}
trained_met={}; trained_prob={}; trained_meta={}

t_total=time.time(); n_ok=0; n_total=len(ALL_RUNS)

for i,(cfg) in enumerate(ALL_RUNS):
    mid=cfg['model_id']; h=int(cfg['horizon']); reg=cfg['regime']
    algo=cfg['algo']; feats=cfg['features']
    ts=cfg.get('train_start','2000-01-01')

    tgt_info=TARGETS[h] if h in TARGETS else None
    if tgt_info is None: continue
    target=tgt_info['target']; reg_s=tgt_info['regime']

    X_mat=build_X(df_features.reindex(target.index),feats)
    if X_mat.empty or X_mat.shape[1]==0: continue
    X_mat=X_mat.copy(); X_mat[TARGET_COL]=target

    try: ts_dt=pd.Timestamp(ts)
    except: ts_dt=pd.Timestamp('2000-01-01')

    mask_tr=(X_mat.index<pd.Timestamp(TEST_DATE))&(X_mat.index>=ts_dt)
    mask_te=(X_mat.index>=pd.Timestamp(TEST_DATE))
    df_tr=X_mat.loc[mask_tr].dropna(subset=[TARGET_COL])
    df_te=X_mat.loc[mask_te].dropna(subset=[TARGET_COL])
    if reg!='GLOBAL':
        df_tr=df_tr.loc[reg_s.reindex(df_tr.index).fillna('NORMAL')==reg]
        df_te=df_te.loc[reg_s.reindex(df_te.index).fillna('NORMAL')==reg]

    fc_=[c for c in X_mat.columns if c!=TARGET_COL]
    if len(df_tr)<30 or len(df_te)<5: continue

    y_tr=df_tr[TARGET_COL].values.astype(int)
    y_te=df_te[TARGET_COL].values.astype(int)
    X_tr=df_tr[fc_].fillna(0).values; X_te=df_te[fc_].fillna(0).values

    sc=RobustScaler()
    X_tr_sc=sc.fit_transform(X_tr); X_te_sc=sc.transform(X_te)
    try:
        samp=get_samp(cfg.get('sampler','SMOTE'))
        X_res,y_res=samp.fit_resample(X_tr_sc,y_tr)
    except: X_res,y_res=X_tr_sc,y_tr

    try:
        clf=get_clf(algo,cfg.get('best_params','{}'))
        clf.fit(X_res,y_res)
    except: continue

    y_pred=clf.predict(X_te_sc)
    y_prob=align4(clf,X_te_sc)
    met=metrics(y_te,y_pred)

    trained_clfs[mid]=clf; trained_sc[mid]=sc; trained_fc[mid]=fc_
    trained_met[mid]={**met,'horizon':h,'regime':reg,'algo':algo,
                      'n_features':int(cfg['n_features']),'F1_dir_ref':float(cfg.get('F1_dir',0.))}
    trained_prob[mid]=y_prob; trained_meta[mid]=cfg; n_ok+=1

    if (i+1)%100==0:
        elapsed=time.time()-t_total
        rate=(i+1)/elapsed; eta=(n_total-i-1)/rate
        print(f"  [{i+1:5d}/{n_total}] OK={n_ok:5d} | "
              f"F1_dir={met['F1_dir']:.4f} | "
              f"Elapsed={elapsed/60:.1f}min ETA={eta/60:.1f}min")

print(f"\n[ML DONE] {n_ok}/{n_total} modèles entraînés en {(time.time()-t_total)/60:.1f}min")


  [  100/9465] OK=  100 | F1_dir=0.5090 | Elapsed=2.0min ETA=184.4min
  [  200/9465] OK=  182 | F1_dir=0.6350 | Elapsed=3.7min ETA=172.4min
  [  300/9465] OK=  276 | F1_dir=0.5638 | Elapsed=4.9min ETA=150.6min
  [  400/9465] OK=  376 | F1_dir=0.5537 | Elapsed=6.6min ETA=150.4min
  [  500/9465] OK=  458 | F1_dir=0.5651 | Elapsed=8.3min ETA=148.8min
  [  600/9465] OK=  557 | F1_dir=0.5750 | Elapsed=11.9min ETA=175.6min
  [  800/9465] OK=  658 | F1_dir=0.5622 | Elapsed=13.2min ETA=142.4min
  [  900/9465] OK=  746 | F1_dir=0.5091 | Elapsed=16.6min ETA=158.4min
  [ 1000/9465] OK=  846 | F1_dir=0.5559 | Elapsed=18.5min ETA=156.4min
  [ 1100/9465] OK=  943 | F1_dir=0.5544 | Elapsed=20.3min ETA=154.7min
  [ 1200/9465] OK= 1043 | F1_dir=0.5224 | Elapsed=23.3min ETA=160.2min
  [ 1400/9465] OK= 1183 | F1_dir=0.4900 | Elapsed=27.2min ETA=156.9min
  [ 1500/9465] OK= 1278 | F1_dir=0.5292 | Elapsed=28.6min ETA=151.7min
  [ 1600/9465] OK= 1358 | F1_dir=0.4395 | Elapsed=30.0min ETA=147.7min
  [ 1800/94

In [ ]:
# ============================================================
# ENTRAÎNEMENT DES MODÈLES DL
# 7 architectures × 6 horizons × 4 régimes = jusqu'à 168 modèles DL
# ============================================================
DL_ARCHS={'LSTM':VIX_LSTM,'TCN':VIX_TCN,'Transformer':VIX_Transformer,
           'CNN-LSTM':VIX_CNNLSTM,'TFT':VIX_TFT,'Mamba':VIX_Mamba}

# Features DL : top SHAP validées (toutes les features TS + marché)
DL_FEATURE_SET=[
    'P_stress_HMM','VIX_Residual','VIX_Innovation',
    'egarch_condvar','egarch_delta','gjr_condvar',
    'heston_xi','heston_theta','heston_kappa','heston_v0',
    'heston_rho','heston_feller','heston_v0_minus_theta',
    'heston_xi_zscore','heston_kappa_zscore',
    'VRP','VRP_zscore','VRP_ma5',
    'jump_intensity_20d','jump_intensity_60d',
    'hawkes_intensity','hawkes_zscore',
    'impl_corr_proxy',
    'vix_vol_of_vol_5d','vix_vol_of_vol_10d',
    'vix_zscore_5d','vix_zscore_10d',
    'vix_momentum_3d','vix_acceleration_1d','vix_acceleration_3d',
    'vix_erratic_ratio','vix_vol_ratio_5_60',
    'vix_max_abs_ret_5d','vix_spx_corr_30d',
    'spx_drawdown_252d','spx_vol_5d',
    'IDX_VIX_ret_1d','IDX_VIX_ret_5d','IDX_VIX_ret_20d',
    'IDX_VIX_vol_20d','IDX_VIX_zscore_60d',
]
# Filtrer les features disponibles
dl_feats=[f for f in DL_FEATURE_SET if f in df_features.columns]
print(f"Features DL disponibles: {len(dl_feats)}/{len(DL_FEATURE_SET)}")

dl_t=time.time()
for h in CONFIG['horizons']:
    for reg in ['GLOBAL','STRESS','CALM','NORMAL']:
        tgt_info=TARGETS[h]; target=tgt_info['target']; reg_s=tgt_info['regime']
        df_al=df_features.reindex(target.index)
        X_base=build_X(df_al,dl_feats).fillna(0)
        X_base[TARGET_COL]=target

        df_tr_dl=X_base.loc[X_base.index<pd.Timestamp(TEST_DATE)].dropna(subset=[TARGET_COL])
        df_te_dl=X_base.loc[X_base.index>=pd.Timestamp(TEST_DATE)].dropna(subset=[TARGET_COL])
        if reg!='GLOBAL':
            df_tr_dl=df_tr_dl.loc[reg_s.reindex(df_tr_dl.index).fillna('NORMAL')==reg]
            df_te_dl=df_te_dl.loc[reg_s.reindex(df_te_dl.index).fillna('NORMAL')==reg]

        if len(df_tr_dl)<100 or len(df_te_dl)<10: continue

        fc_dl=[c for c in X_base.columns if c!=TARGET_COL]
        y_tr_dl=df_tr_dl[TARGET_COL].values.astype(int)
        y_te_dl=df_te_dl[TARGET_COL].values.astype(int)

        sc_dl=RobustScaler()
        X_tr_dl=sc_dl.fit_transform(df_tr_dl[fc_dl].fillna(0).values)
        X_te_dl=sc_dl.transform(df_te_dl[fc_dl].fillna(0).values)

        try:
            samp=BorderlineSMOTE(random_state=SEED)
            X_res_dl,y_res_dl=samp.fit_resample(X_tr_dl,y_tr_dl)
        except: X_res_dl,y_res_dl=X_tr_dl,y_tr_dl

        cw=compute_cw(y_res_dl)
        val_sp=int(len(X_res_dl)*0.85)
        ds_tr=PreScaledDS(X_res_dl[:val_sp],y_res_dl[:val_sp])
        ds_va=PreScaledDS(X_res_dl[val_sp:],y_res_dl[val_sp:])
        ds_te=PreScaledDS(X_te_dl,y_te_dl)
        dl_tr_dl=DataLoader(ds_tr,batch_size=CONFIG['dl_batch'],shuffle=True)
        dl_va_dl=DataLoader(ds_va,batch_size=256)
        dl_te_dl=DataLoader(ds_te,batch_size=256)

        input_dim=len(fc_dl)
        for arch_name,ArchClass in DL_ARCHS.items():
            mid_dl=f"DL_h{h}_{reg}_{arch_name}"
            t0=time.time()
            try:
                model=ArchClass(input_dim).to(device)
                train_dl(model,dl_tr_dl,dl_va_dl,cw=cw,label=mid_dl,patience=8)
                tgt_arr,pred_arr,prob_arr=eval_dl(model,dl_te_dl)
                met_dl=metrics(tgt_arr,pred_arr)
                trained_clfs[mid_dl]=model; trained_sc[mid_dl]=sc_dl
                trained_fc[mid_dl]=fc_dl; trained_prob[mid_dl]=prob_arr
                trained_met[mid_dl]={**met_dl,'horizon':h,'regime':reg,'algo':arch_name,
                                      'n_features':input_dim,'is_dl':True}
                trained_meta[mid_dl]={'horizon':h,'regime':reg,'algo':arch_name,
                                       'features':fc_dl,'sampler':'BorderlineSMOTE',
                                       'best_params':'{}','train_start':'2000-01-01','is_dl':True}
                print(f"  {mid_dl} F1_dir={met_dl['F1_dir']:.4f} ({time.time()-t0:.0f}s)")
            except Exception as e:
                print(f"  [WARN] {mid_dl}: {e}")

n_dl=[m for m in trained_met if trained_met[m].get('is_dl',False)]
print(f"\n[DL DONE] {len(n_dl)} modèles DL en {(time.time()-dl_t)/60:.1f}min")
print(f"Total panel (ML+DL): {len(trained_clfs)} modèles")


In [ ]:
# ============================================================
# RE-SÉLECTION PAR CORRÉLATION DES PRÉDICTIONS
# + GÉNÉRATION DES OOF + MÉTA-MODÈLES
# ============================================================
print(f"=== Panel total: {len(trained_clfs)} modèles ===")

# Matrice de corrélation inter-prédictions
model_ids=list(trained_prob.keys())
dir_preds={mid:(p[:,2]+p[:,3]>0.5).astype(int) for mid,p in trained_prob.items()}
min_len=min(len(v) for v in dir_preds.values())
dir_mat=np.column_stack([dir_preds[m][:min_len] for m in model_ids])
corr_mat=np.corrcoef(dir_mat.T)
upper=corr_mat[np.triu_indices_from(corr_mat,k=1)]
print(f"Corrélation inter-prédictions: mean={upper.mean():.3f} >0.85={(upper>0.85).mean():.1%}")

# Sélection greedy par diversité
def select_diverse(model_ids,dir_preds,metrics_dict,max_corr=0.85):
    sel=[]; sel_dir=[]
    for mid in sorted(model_ids,key=lambda x:metrics_dict.get(x,{}).get('F1_dir',0),reverse=True):
        d=dir_preds[mid][:min_len]
        if not sel: sel.append(mid); sel_dir.append(d); continue
        corrs=[np.corrcoef(d,s[:len(d)])[0,1] for s in sel_dir]
        if max(corrs)<max_corr: sel.append(mid); sel_dir.append(d)
    return sel

final_panel=select_diverse(model_ids,dir_preds,trained_met,CONFIG['max_pred_corr'])
print(f"Panel diversifié: {len(final_panel)} modèles (corr<{CONFIG['max_pred_corr']})")

# OOF pour le stacking
META_H=5
tgt_meta=TARGETS[META_H]['target']
tr_dates=tgt_meta.index[tgt_meta.index<pd.Timestamp(TEST_DATE)]
te_dates=tgt_meta.index[tgt_meta.index>=pd.Timestamp(TEST_DATE)]
y_tr_dir=(tgt_meta.reindex(tr_dates).fillna(0).values>=2).astype(int)
y_te_dir=(tgt_meta.reindex(te_dates).fillna(0).values>=2).astype(int)
y_tr_4=tgt_meta.reindex(tr_dates).fillna(0).values.astype(int)
y_te_4=tgt_meta.reindex(te_dates).fillna(0).values.astype(int)

def gen_oof(mid,n_folds=5):
    cfg=trained_meta[mid]; is_dl=cfg.get('is_dl',False)
    h=int(cfg['horizon']); reg=str(cfg['regime'])
    feats=cfg['features']
    tgt_h=TARGETS[h]['target']; reg_s=TARGETS[h]['regime']
    X_mat=build_X(df_features.reindex(tgt_h.index),feats)
    if X_mat.empty: return None,None
    X_mat=X_mat.copy(); X_mat[TARGET_COL]=tgt_h
    df_tr2=X_mat.loc[X_mat.index<pd.Timestamp(TEST_DATE)].dropna(subset=[TARGET_COL])
    if reg!='GLOBAL': df_tr2=df_tr2.loc[reg_s.reindex(df_tr2.index).fillna('NORMAL')==reg]
    if len(df_tr2)<50: return None,None
    fc2=[c for c in X_mat.columns if c!=TARGET_COL]
    sc=trained_sc[mid]; X_t=sc.transform(df_tr2[fc2].fillna(0).values); y_t=df_tr2[TARGET_COL].values.astype(int)
    oof=np.full((len(X_t),4),0.25); tscv=TimeSeriesSplit(n_splits=n_folds)
    for ti,vi in tscv.split(X_t):
        if len(vi)<5: continue
        if is_dl:
            # Pour les modèles DL, utiliser directement les proba globales (pas d'OOF complet)
            oof[vi]=0.25; continue
        try:
            sm=get_samp(cfg.get('sampler','SMOTE')); Xr,yr=sm.fit_resample(X_t[ti],y_t[ti])
        except: Xr,yr=X_t[ti],y_t[ti]
        try:
            c=get_clf(cfg['algo'],cfg.get('best_params','{}')); c.fit(Xr,yr)
            oof[vi]=align4(c,X_t[vi])
        except: pass
    return oof,df_tr2.index

print("\nGénération OOF...")
t0=time.time(); meta_tr=[]; meta_te=[]; used=[]
for i,mid in enumerate(final_panel):
    oof,oof_dates=gen_oof(mid)
    if oof is None: continue
    cols=[f'{mid}_c{k}' for k in range(4)]
    meta_tr.append(pd.DataFrame(oof,index=oof_dates,columns=cols).reindex(tr_dates).fillna(0.25))
    proba_te=trained_prob[mid]
    cfg_r=trained_meta[mid]; h_m=int(cfg_r['horizon']); reg_m=str(cfg_r['regime'])
    te_h=TARGETS[h_m]['target'].index; te_h=te_h[te_h>=pd.Timestamp(TEST_DATE)]
    if reg_m!='GLOBAL': te_h=te_h[TARGETS[h_m]['regime'].reindex(te_h).fillna('NORMAL')==reg_m]
    n_m=min(len(proba_te),len(te_h))
    meta_te.append(pd.DataFrame(proba_te[:n_m],index=te_h[:n_m],columns=cols).reindex(te_dates).fillna(0.25))
    used.append(mid)
    if (i+1)%20==0: print(f"  OOF [{i+1}/{len(final_panel)}] ({time.time()-t0:.0f}s)")

meta_X_tr=pd.concat(meta_tr,axis=1).fillna(0.25)
meta_X_te=pd.concat(meta_te,axis=1).fillna(0.25)
print(f"\nMéta: train={meta_X_tr.shape} test={meta_X_te.shape} ({time.time()-t0:.0f}s)")

# Méta-modèles
print("\nEntraînement méta-modèles...")
meta_dir=XGBClassifier(n_estimators=CONFIG['meta_n_estimators'],max_depth=CONFIG['meta_max_depth'],
    learning_rate=CONFIG['meta_lr'],subsample=0.8,colsample_bytree=0.8,
    eval_metric='logloss',random_state=SEED,n_jobs=-1,verbosity=0)
meta_dir.fit(meta_X_tr.values,y_tr_dir)
y_pred_dir=meta_dir.predict(meta_X_te.values)
f1_dir=f1_score(y_te_dir,y_pred_dir,average='macro',zero_division=0)
acc_dir=accuracy_score(y_te_dir,y_pred_dir)

meta_4cls=XGBClassifier(n_estimators=CONFIG['meta_n_estimators'],max_depth=CONFIG['meta_max_depth'],
    learning_rate=CONFIG['meta_lr'],subsample=0.8,colsample_bytree=0.8,
    eval_metric='mlogloss',objective='multi:softprob',random_state=SEED,n_jobs=-1,verbosity=0)
meta_4cls.fit(meta_X_tr.values,y_tr_4)
y_pred_4=meta_4cls.predict(meta_X_te.values)
met_4=metrics(y_te_4,y_pred_4)

# Vote pondéré
su=np.zeros(len(meta_X_te)); sd_=np.zeros(len(meta_X_te))
for mid in used:
    f1w=trained_met.get(mid,{}).get('F1_dir',0.5)
    for k in [2,3]:
        c=f'{mid}_c{k}'
        if c in meta_X_te.columns: su+=f1w*meta_X_te[c].values
    for k in [0,1]:
        c=f'{mid}_c{k}'
        if c in meta_X_te.columns: sd_+=f1w*meta_X_te[c].values
y_vote=(su>sd_).astype(int)
f1_vote=f1_score(y_te_dir,y_vote,average='macro',zero_division=0)

print(f"\nVote pondéré      F1_dir={f1_vote:.4f}")
print(f"Méta XGB dir      F1_dir={f1_dir:.4f}  Acc={acc_dir:.4f}")
print(f"Méta XGB 4cls     F1_dir={met_4['F1_dir']:.4f}  UP_FORT={met_4['F1_UP_FORT']:.4f}  DOWN_FORT={met_4['F1_DOWN_FORT']:.4f}")


In [ ]:
# ============================================================
# SHAP + RAPPORT FINAL + EXPORT
# ============================================================
try:
    expl=shap.TreeExplainer(meta_4cls)
    sv=expl.shap_values(meta_X_te.values[:min(300,len(meta_X_te))])
    if isinstance(sv,list): arr=np.mean([np.abs(s) for s in sv],axis=0)
    elif np.array(sv).ndim==3: arr=np.abs(sv).mean(axis=2)
    else: arr=np.abs(sv)
    shap_scores=pd.Series(arr.mean(axis=0),index=meta_X_te.columns)
    top_shap=shap_scores.nlargest(40)
    print("Top-20 features SHAP méta-modèle:")
    for feat,score in top_shap.head(20).items():
        mid_s='_c'.join(feat.split('_c')[:-1])
        info=trained_met.get(mid_s,{})
        is_dl=trained_meta.get(mid_s,{}).get('is_dl',False)
        tag='DL' if is_dl else 'ML'
        print(f"  [{tag}] {feat[:55]:<55} {score:.5f} (h={info.get('horizon','?')}j {info.get('regime','?')} F1={info.get('F1_dir',0):.3f})")
except Exception as e: print(f"[WARN SHAP] {e}"); top_shap=pd.Series(dtype=float)

print("\n"+"="*75)
n_ml=sum(1 for m in used if not trained_meta.get(m,{}).get('is_dl',False))
n_dl_u=sum(1 for m in used if trained_meta.get(m,{}).get('is_dl',False))
print(f"RAPPORT FINAL — {len(used)} modèles ({n_ml} ML + {n_dl_u} DL)")
print(f"Cible méta: h={META_H}j | Split 80/20")
print("="*75)
print(f"  Vote pondéré (ML+DL)  F1_dir={f1_vote:.4f}")
print(f"  Méta XGB direction    F1_dir={f1_dir:.4f}  Acc={acc_dir:.4f}")
print(f"  Méta XGB 4 classes    F1_dir={met_4['F1_dir']:.4f}  UP_FORT={met_4['F1_UP_FORT']:.4f}  DOWN_FORT={met_4['F1_DOWN_FORT']:.4f}")
print("─"*75)
print("  Benchmarks ML:")
for k,v in ML_REFS.items():
    print(f"  {k:<45} F1_dir={v['F1_dir']:.4f}")
print("  Benchmarks DL:")
for hh,v in DL_REFS.items():
    print(f"  DL best h={hh}j          F1_dir={v['F1_dir']:.4f}  UP_FORT={v['F1_UP_FORT']:.4f}")

try:
    rows=[{'Model_ID':m,**trained_met[m],'is_dl':trained_meta.get(m,{}).get('is_dl',False)} for m in used if m in trained_met]
    df_out=pd.DataFrame(rows).sort_values('F1_dir',ascending=False)
    meta_rows=[
        {'Model_ID':'Vote_Pondéré','F1_dir':f1_vote,'N_ML':n_ml,'N_DL':n_dl_u},
        {'Model_ID':'Meta_XGB_dir','F1_dir':f1_dir,'Acc_dir':acc_dir,'N_total':len(used)},
        {'Model_ID':'Meta_XGB_4cls',**met_4,'N_total':len(used)},
    ]
    with pd.ExcelWriter('VIX_ML3_stacking_report.xlsx',engine='xlsxwriter') as w:
        df_out.to_excel(w,'Panel_Models',index=False)
        pd.DataFrame(meta_rows).to_excel(w,'Stacking_Results',index=False)
        if not top_shap.empty:
            top_shap.reset_index().rename(columns={'index':'Feature',0:'SHAP'}).to_excel(w,'SHAP_Meta',index=False)
        df_out.groupby(['horizon','regime','algo'])['F1_dir'].agg(['count','mean','max']).reset_index().to_excel(w,'Stats_Panel',index=False)
    print("\n[SAVE] VIX_ML3_stacking_report.xlsx")
except Exception as e: print(f"[WARN Export] {e}")
print("[NOTE] Aucun modèle enregistré — validation explicite requise.")
